# 04 - Baseline Models Evaluation (B0, A0, B3 Pilot)

**Đồ án môn Học máy:** Phân loại 4 nhóm ảnh X-quang ngực (COVID-19 Radiography Database).  
**Sprint 2 - Tuần 4:** Thiết lập các mốc mô hình cơ sở trên tập **Validation** (tuyệt đối KHÔNG đụng tập Test):
- **B0:** Mô hình luôn đoán lớp đa số (`Normal`).
- **A0:** Mô hình phi bệnh lý (đo lường lối tắt phân loại dựa trên độ sáng và kênh màu).
- **B3 Pilot:** Thử nghiệm tiền trạm mô hình mạng nơ-ron tích chập ResNet18 (Full Fine-tuning, Seed 42, trọng số ImageNet official) để đo đạc thời gian và GPU-giờ tiêu hao.

> **Quy tắc nghiêm ngặt:** Chỉ đánh giá trên tập Validation; `allow_test` luôn là `False`; không xử lý mất cân bằng ở giai đoạn này.

In [ ]:
# 1. Khởi tạo môi trường, module src/ và nạp các tệp dữ liệu phân chia
import os
import sys
import time
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torchvision

print(f"PyTorch Version: {torch.__version__}")
print(f"Torchvision Version: {torchvision.__version__}")
print(f"CUDA khả dụng: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Thiết bị GPU: {torch.cuda.get_device_name(0)}")
    print(f"Số lượng GPU: {torch.cuda.device_count()}")

os.makedirs("src", exist_ok=True)
os.makedirs("splits", exist_ok=True)
os.makedirs("results/b0_majority", exist_ok=True)
os.makedirs("results/a0_shortcut", exist_ok=True)
os.makedirs("results/pilot_b3", exist_ok=True)
os.makedirs("outputs/checkpoints", exist_ok=True)

with open("src/__init__.py", "w", encoding="utf-8") as f: f.write("")
with open("src/dataset.py", "w", encoding="utf-8") as f: f.write('''"""Module dataset.py

Trách nhiệm:
- Quản lý nạp dữ liệu từ file split_v1.csv (phụ trách: Nguyễn Đức).
- Định nghĩa lớp PyTorch CXRDataset cho bài toán phân loại 4 nhóm ảnh X-quang ngực.
- Thiết kế Data Augmentation cho tập Train và Transform chuẩn hóa cho Validation (phụ trách: Vũ Thương).
- Cài đặt cơ chế bảo vệ nghiêm ngặt: Khóa tập Test (allow_test=False).
- Tính toán Class Weights hỗ trợ xử lý mất cân bằng lớp.
"""

import os
from pathlib import Path
from typing import Callable, Dict, List, Optional, Tuple, Union

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

# 4 lớp bài toán phân loại X-quang ngực
CLASS_NAMES = ["COVID", "Lung_Opacity", "Normal", "Viral Pneumonia"]
LABEL_TO_IDX: Dict[str, int] = {name: idx for idx, name in enumerate(CLASS_NAMES)}
IDX_TO_LABEL: Dict[int, str] = {idx: name for idx, name in enumerate(CLASS_NAMES)}

# Chuẩn hóa ImageNet
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def get_transforms(
    img_size: int = 224,
    is_train: bool = True,
) -> transforms.Compose:
    """Tạo chuỗi biến đổi tiền xử lý và tăng cường dữ liệu ảnh X-quang.

    Quy tắc y khoa nghiêm ngặt (Vũ Thương phụ trách):
    - TUYỆT ĐỐI KHÔNG sử dụng RandomHorizontalFlip (lật ngang).
      Lý do: Tim người nằm ở lồng ngực bên trái; lật ngang làm đảo ngược vị trí tim và
      bóng các cơ quan nội tạng (Situs Inversus nhân tạo), gây sai lệch đặc trưng y học.
    - Train augmentation an toàn: Xoay nhẹ (+/- 7 độ), co giãn nhẹ (scale 0.95-1.05),
      chỉnh sáng và tương phản nhẹ (+/- 10%).
    - Validation: Chỉ resize và normalize.

    Args:
        img_size (int): Kích thước cạnh ảnh đầu ra (mặc định 224).
        is_train (bool): True nếu áp dụng augmentation cho tập train, False cho val/test.

    Returns:
        transforms.Compose: Pipeline tiền xử lý của torchvision.
    """
    if is_train:
        return transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomRotation(degrees=(-7, 7)),
            transforms.RandomAffine(
                degrees=0,
                translate=(0.02, 0.02),
                scale=(0.95, 1.05),
            ),
            transforms.ColorJitter(brightness=0.1, contrast=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])
    else:
        return transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])


class CXRDataset(Dataset):
    """Lớp Dataset nạp ảnh X-quang ngực từ split_v1.csv.

    Hỗ trợ cơ chế khóa tập test: mặc định cấm truy cập split='test' (allow_test=False).
    """

    def __init__(
        self,
        split_df_or_path: Union[pd.DataFrame, str, Path],
        split: str = "train",
        data_root: Optional[Union[str, Path]] = None,
        transform: Optional[Callable] = None,
        allow_test: bool = False,
    ) -> None:
        """Khởi tạo CXRDataset.

        Args:
            split_df_or_path: Đường dẫn tới file split_v1.csv hoặc pandas DataFrame.
            split (str): Phân vùng cần nạp ('train', 'val', 'test').
            data_root: Thư mục gốc chứa ảnh (ví dụ /kaggle/input/.../COVID-19_Radiography_Dataset).
            transform: Chuỗi biến đổi ảnh. Nếu None sẽ tự tạo theo get_transforms.
            allow_test (bool): Cờ cho phép đọc tập test. Mặc định là False.
        """
        split_lower = split.lower().strip()
        if split_lower not in {"train", "val", "test"}:
            raise ValueError(f"Split không hợp lệ: '{split}'. Chỉ chấp nhận 'train', 'val', hoặc 'test'.")

        # Cơ chế khóa nghiêm ngặt tập Test
        if split_lower == "test" and not allow_test:
            raise PermissionError(
                "TẬP TEST ĐÃ BỊ KHÓA NGHIÊM NGẶT!\\n"
                "Theo quy tắc đạo đức nghiên cứu và chống rò rỉ dữ liệu của đồ án:\\n"
                "- Không được phép truy cập, đánh giá hoặc tinh chỉnh siêu tham số trên tập test ở Tuần 3-9.\\n"
                "- Mọi quyết định lựa chọn mô hình chỉ được thực hiện trên tập Validation.\\n"
                "- Tập Test chỉ được mở DUY NHẤT một lần tại Tuần 10 trên mô hình tối ưu đã chốt."
            )

        if isinstance(split_df_or_path, (str, Path)):
            df_full = pd.read_csv(split_df_or_path)
        else:
            df_full = split_df_or_path.copy()

        self.df = df_full[df_full["split"] == split_lower].reset_index(drop=True)
        self.split = split_lower
        self.data_root = Path(data_root) if data_root is not None else None

        if transform is not None:
            self.transform = transform
        else:
            self.transform = get_transforms(img_size=224, is_train=(split_lower == "train"))

    def __len__(self) -> int:
        return len(self.df)

    def _resolve_image_path(self, row: pd.Series) -> Path:
        """Định vị đường dẫn ảnh tuyệt đối một cách linh hoạt."""
        # 1. Thử qua data_root ghép với rel_image_path
        if self.data_root is not None:
            candidate = self.data_root / row["rel_image_path"]
            if candidate.exists():
                return candidate

        # 2. Thử rel_image_path trực tiếp từ thư mục hiện tại
        candidate = Path(row["rel_image_path"])
        if candidate.exists():
            return candidate

        # 3. Thử image_path lưu trong CSV nếu có
        if "image_path" in row and pd.notna(row["image_path"]):
            candidate = Path(row["image_path"])
            if candidate.exists():
                return candidate

        raise FileNotFoundError(
            f"Không thể tìm thấy file ảnh: {row['image_id']} (rel: {row['rel_image_path']})"
        )

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        row = self.df.iloc[idx]
        img_path = self._resolve_image_path(row)

        with Image.open(img_path) as img:
            # Luôn chuyển đổi sang RGB 3 kênh để tương thích các kiến trúc CNN
            img = img.convert("RGB")
            tensor_img = self.transform(img)

        label_name = row["label"]
        label_idx = LABEL_TO_IDX[label_name]

        return tensor_img, label_idx


def compute_class_weights(
    split_df_or_path: Union[pd.DataFrame, str, Path],
    split: str = "train",
    num_classes: int = 4,
) -> torch.FloatTensor:
    """Tính toán trọng số nghịch đảo tần suất lớp (Balanced Class Weights) cho Loss Function.

    Công thức: w_c = N / (C * N_c)
    Giúp giảm thiểu tác động tiêu cực của tình trạng mất cân bằng lớp (Normal chiếm ~48%).

    Args:
        split_df_or_path: Đường dẫn tới CSV hoặc DataFrame.
        split (str): Mặc định tính trên tập 'train'.
        num_classes (int): Số lớp (4).

    Returns:
        torch.FloatTensor: Mảng trọng số 1D tensor [w_0, w_1, w_2, w_3].
    """
    if isinstance(split_df_or_path, (str, Path)):
        df = pd.read_csv(split_df_or_path)
    else:
        df = split_df_or_path

    df_subset = df[df["split"] == split.lower()]
    total_samples = len(df_subset)

    class_counts = df_subset["label"].value_counts().to_dict()
    weights = np.zeros(num_classes, dtype=np.float32)

    for cls_name, cls_idx in LABEL_TO_IDX.items():
        count = class_counts.get(cls_name, 1)
        weights[cls_idx] = total_samples / (num_classes * count)

    return torch.tensor(weights, dtype=torch.float32)


def build_dataloaders(
    split_csv_path: Union[str, Path],
    data_root: Optional[Union[str, Path]] = None,
    batch_size: int = 32,
    num_workers: int = 2,
    img_size: int = 224,
    allow_test: bool = False,
) -> Dict[str, DataLoader]:
    """Khởi tạo DataLoaders cho các phân vùng huấn luyện và kiểm định.

    Args:
        split_csv_path: Đường dẫn tới file split_v1.csv.
        data_root: Thư mục gốc chứa ảnh (nếu chạy trên Kaggle/Colab).
        batch_size: Kích thước mini-batch.
        num_workers: Số luồng nạp dữ liệu.
        img_size: Kích thước ảnh đầu vào mô hình.
        allow_test: Cờ cho phép nạp tập test (mặc định False).

    Returns:
        Dict[str, DataLoader]: Dict chứa 'train', 'val' (và 'test' nếu allow_test=True).
    """
    train_ds = CXRDataset(
        split_df_or_path=split_csv_path,
        split="train",
        data_root=data_root,
        transform=get_transforms(img_size=img_size, is_train=True),
        allow_test=False,
    )

    val_ds = CXRDataset(
        split_df_or_path=split_csv_path,
        split="val",
        data_root=data_root,
        transform=get_transforms(img_size=img_size, is_train=False),
        allow_test=False,
    )

    dataloaders = {
        "train": DataLoader(
            train_ds,
            batch_size=batch_size,
            shuffle=True,
            num_workers=num_workers,
            pin_memory=torch.cuda.is_available(),
        ),
        "val": DataLoader(
            val_ds,
            batch_size=batch_size,
            shuffle=False,
            num_workers=num_workers,
            pin_memory=torch.cuda.is_available(),
        ),
    }

    if allow_test:
        test_ds = CXRDataset(
            split_df_or_path=split_csv_path,
            split="test",
            data_root=data_root,
            transform=get_transforms(img_size=img_size, is_train=False),
            allow_test=True,
        )
        dataloaders["test"] = DataLoader(
            test_ds,
            batch_size=batch_size,
            shuffle=False,
            num_workers=num_workers,
            pin_memory=torch.cuda.is_available(),
        )

    return dataloaders
''')
with open("src/models.py", "w", encoding="utf-8") as f: f.write('''"""Module models.py

Trách nhiệm:
- Định nghĩa và khởi tạo kiến trúc mạng học sâu cho bài toán phân loại X-quang 4 lớp.
- Hỗ trợ mô hình ResNet-18 (tối thiểu cho Smoke Test Tuần 3), mở rộng DenseNet-121 và EfficientNet-B0.
- Cho phép khởi tạo không cần trọng số pretrained (pretrained=False) để chạy offline trên Kaggle khi tắt internet.
- Thay thế classifier head tương ứng với 4 lớp phân loại.
- Cung cấp tiện ích đóng băng (freeze) và mở khóa (unfreeze) backbone phục vụ transfer learning.

Ghi chú phân công:
- Theo docs/OWNERS.md, src/models.py do Thái Tú (TV3) phụ trách chính, Vũ Thương (TV4) kèm cặp duyệt chéo.
"""

from typing import Optional
import torch
import torch.nn as nn
from torchvision import models


def get_model(
    model_name: str = "resnet18",
    num_classes: int = 4,
    pretrained: bool = False,
) -> nn.Module:
    """Khởi tạo mô hình CNN và thay thế classifier head cho bài toán 4 lớp.

    Args:
        model_name (str): Tên kiến trúc ('resnet18', 'densenet121', 'efficientnet_b0').
        num_classes (int): Số lượng lớp phân loại (mặc định 4).
        pretrained (bool): True để tải trọng số ImageNet (yêu cầu internet hoặc file weights có sẵn).
                           False để khởi tạo trọng số ngẫu nhiên (dùng cho Smoke Test với internet=False).

    Returns:
        nn.Module: Mô hình PyTorch đã điều chỉnh classifier head.
    """
    name_clean = model_name.lower().replace("-", "").replace("_", "")

    if "resnet18" in name_clean:
        weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        model = models.resnet18(weights=weights)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)

    elif "densenet121" in name_clean:
        weights = models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None
        model = models.densenet121(weights=weights)
        in_features = model.classifier.in_features
        model.classifier = nn.Linear(in_features, num_classes)

    elif "efficientnetb0" in name_clean:
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
        model = models.efficientnet_b0(weights=weights)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)

    else:
        raise ValueError(
            f"Kiến trúc '{model_name}' chưa được hỗ trợ! "
            f"Các tùy chọn hợp lệ: ['resnet18', 'densenet121', 'efficientnet_b0']."
        )

    return model


def freeze_backbone(model: nn.Module) -> None:
    """Đóng băng toàn bộ trọng số của backbone (chỉ huấn luyện classifier head)."""
    for param in model.parameters():
        param.requires_grad = False

    # Mở khóa head phân loại
    if hasattr(model, "fc"):
        for param in model.fc.parameters():
            param.requires_grad = True
    elif hasattr(model, "classifier"):
        for param in model.classifier.parameters():
            param.requires_grad = True


def unfreeze_all(model: nn.Module) -> None:
    """Mở khóa toàn bộ các tham số trong mạng (fine-tuning toàn bộ mô hình)."""
    for param in model.parameters():
        param.requires_grad = True
''')
with open("src/metrics.py", "w", encoding="utf-8") as f: f.write('''"""Module metrics.py

Trách nhiệm:
- Tính toán toàn diện các chỉ số đánh giá cho bài toán phân loại X-quang 4 lớp (phụ trách: Hải Lý).
- Chỉ số chính: Macro-F1 (tiêu chí chính để chọn mô hình và lưu best checkpoint), Balanced Accuracy.
- Chỉ số chi tiết: Accuracy, Weighted-F1, Per-class Precision, Per-class Recall, Per-class F1.
- Xuất ma trận nhầm lẫn (Confusion Matrix) dạng số liệu và biểu đồ trực quan hóa.
"""

from typing import Dict, List, Optional, Union
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

DEFAULT_CLASS_NAMES = ["COVID", "Lung_Opacity", "Normal", "Viral Pneumonia"]


def compute_metrics(
    y_true: Union[List[int], np.ndarray],
    y_pred: Union[List[int], np.ndarray],
    class_names: Optional[List[str]] = None,
) -> Dict[str, Union[float, Dict[str, float], List[List[int]]]]:
    """Tính toán toàn bộ các thước đo hiệu năng phân loại đa lớp.

    Args:
        y_true: Danh sách hoặc mảng nhãn thực tế (ground truth).
        y_pred: Danh sách hoặc mảng nhãn mô hình dự đoán.
        class_names: Danh sách tên các lớp (mặc định 4 lớp).

    Returns:
        Dict chứa:
        - 'accuracy': float
        - 'balanced_accuracy': float
        - 'macro_f1': float (thước đo chính)
        - 'weighted_f1': float
        - 'macro_precision': float
        - 'macro_recall': float
        - 'per_class': Dict tên lớp -> {'precision', 'recall', 'f1', 'support'}
        - 'confusion_matrix': List[List[int]]
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    if class_names is None:
        class_names = DEFAULT_CLASS_NAMES
    num_classes = len(class_names)

    acc = float(accuracy_score(y_true, y_pred))
    balanced_acc = float(balanced_accuracy_score(y_true, y_pred))
    macro_f1 = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    weighted_f1 = float(f1_score(y_true, y_pred, average="weighted", zero_division=0))
    macro_prec = float(precision_score(y_true, y_pred, average="macro", zero_division=0))
    macro_rec = float(recall_score(y_true, y_pred, average="macro", zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))

    # Per-class metrics
    per_class_prec = precision_score(y_true, y_pred, average=None, labels=list(range(num_classes)), zero_division=0)
    per_class_rec = recall_score(y_true, y_pred, average=None, labels=list(range(num_classes)), zero_division=0)
    per_class_f1 = f1_score(y_true, y_pred, average=None, labels=list(range(num_classes)), zero_division=0)

    per_class_dict = {}
    for idx, name in enumerate(class_names):
        support_count = int(np.sum(y_true == idx))
        per_class_dict[name] = {
            "precision": float(per_class_prec[idx]),
            "recall": float(per_class_rec[idx]),
            "f1": float(per_class_f1[idx]),
            "support": support_count,
        }

    return {
        "accuracy": round(acc, 4),
        "balanced_accuracy": round(balanced_acc, 4),
        "macro_f1": round(macro_f1, 4),
        "weighted_f1": round(weighted_f1, 4),
        "macro_precision": round(macro_prec, 4),
        "macro_recall": round(macro_rec, 4),
        "per_class": per_class_dict,
        "confusion_matrix": cm.tolist(),
    }


def plot_confusion_matrix(
    cm: Union[List[List[int]], np.ndarray],
    class_names: Optional[List[str]] = None,
    save_path: Optional[str] = None,
    normalize: bool = False,
    title: str = "Confusion Matrix",
) -> plt.Figure:
    """Vẽ và lưu biểu đồ ma trận nhầm lẫn bằng Matplotlib.

    Args:
        cm: Ma trận nhầm lẫn dạng 2D array hoặc list.
        class_names: Danh sách tên lớp hiển thị trên các trục.
        save_path: Đường dẫn lưu file ảnh (tùy chọn).
        normalize: True nếu muốn chuẩn hóa tỷ lệ theo hàng (Recall per class).
        title: Tiêu đề biểu đồ.

    Returns:
        plt.Figure: Đối tượng Figure của matplotlib.
    """
    cm = np.asarray(cm)
    if class_names is None:
        class_names = DEFAULT_CLASS_NAMES

    if normalize:
        row_sums = cm.sum(axis=1)[:, np.newaxis]
        row_sums[row_sums == 0] = 1
        cm_display = cm.astype("float") / row_sums
        fmt = ".2%"
    else:
        cm_display = cm
        fmt = "d"

    fig, ax = plt.subplots(figsize=(7, 6))
    cax = ax.matshow(cm_display, cmap="Blues")
    fig.colorbar(cax)

    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=30, ha="left", fontsize=9)
    ax.set_yticklabels(class_names, fontsize=9)
    ax.set_xlabel("Predicted Label", fontweight="bold", labelpad=10)
    ax.set_ylabel("True Label", fontweight="bold", labelpad=10)
    ax.set_title(title, fontweight="bold", pad=15)

    thresh = cm_display.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            val_str = f"{cm_display[i, j]:{fmt}}"
            color = "white" if cm_display[i, j] > thresh else "black"
            ax.text(j, i, val_str, ha="center", va="center", color=color, fontsize=9)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Đã lưu ma trận nhầm lẫn tại: {save_path}")

    return fig
''')
with open("src/train.py", "w", encoding="utf-8") as f: f.write('''"""Module train.py

Trách nhiệm:
- Điều khiển vòng lặp huấn luyện (train loop) và kiểm định (val loop) cho mô hình (phụ trách: Minh Thuận).
- Hỗ trợ huấn luyện độ chính xác hỗn hợp (Mixed Precision) với torch.cuda.amp giúp tối ưu bộ nhớ GPU.
- Quản lý checkpoint nghiêm ngặt:
  + Tự động lưu 'last.pt' sau mỗi epoch để sẵn sàng phục hồi khi kernel bị ngắt phiên (preemption).
  + Tự động lưu 'best.pt' khi chỉ số Macro-F1 trên tập Validation đạt kỷ lục mới.
- Cơ chế Resume Training liền mạch: Nạp model, optimizer, scheduler, scaler và tiếp tục từ epoch tiếp theo.
- Quy ước đặt tên run chuẩn hóa phục vụ quản lý thực nghiệm.
"""

from datetime import datetime
import json
import os
from pathlib import Path
import time
from typing import Any, Dict, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from src.metrics import compute_metrics


def generate_run_id(model_name: str = "resnet18", seed: int = 42) -> str:
    """Tạo mã định danh duy nhất cho một lượt huấn luyện (run_id).

    Quy ước đặt tên: <model_name>_seed<seed>_<YYYYMMDD_HHMMSS>
    Ví dụ: resnet18_seed42_20261002_203000
    """
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    clean_model = model_name.lower().replace("-", "").replace("_", "")
    return f"{clean_model}_seed{seed}_{timestamp}"


def save_checkpoint(
    state: Dict[str, Any],
    filepath: Path,
) -> None:
    """Lưu checkpoint PyTorch an toàn vào đĩa."""
    filepath.parent.mkdir(parents=True, exist_ok=True)
    torch.save(state, str(filepath))


def load_checkpoint(
    checkpoint_path: Path,
    model: nn.Module,
    optimizer: Optional[torch.optim.Optimizer] = None,
    scheduler: Optional[Any] = None,
    scaler: Optional[Any] = None,
    device: str = "cpu",
) -> Tuple[int, float]:
    """Khôi phục trạng thái huấn luyện từ checkpoint .pt để resume phiên làm việc.

    Args:
        checkpoint_path: Đường dẫn tới file checkpoint (.pt).
        model: Mô hình PyTorch cần nạp trọng số.
        optimizer: Bộ tối ưu cần nạp trạng thái (tùy chọn).
        scheduler: Bộ điều chỉnh learning rate (tùy chọn).
        scaler: GradScaler của AMP (tùy chọn).
        device: Thiết bị tính toán ('cuda' hoặc 'cpu').

    Returns:
        Tuple[int, float]: (start_epoch, best_val_f1)
        start_epoch: Epoch tiếp theo cần chạy (epoch đã lưu + 1).
        best_val_f1: Điểm Macro-F1 tốt nhất từng ghi nhận.
    """
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Không tìm thấy file checkpoint để resume: {checkpoint_path}")

    checkpoint = torch.load(str(checkpoint_path), map_location=device)

    # 1. Nạp trọng số mô hình
    model.load_state_dict(checkpoint["model_state_dict"])

    # 2. Nạp trạng thái optimizer nếu có
    if optimizer is not None and "optimizer_state_dict" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    # 3. Nạp trạng thái scheduler nếu có
    if scheduler is not None and "scheduler_state_dict" in checkpoint and checkpoint["scheduler_state_dict"] is not None:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    # 4. Nạp trạng thái AMP scaler nếu có
    if scaler is not None and "scaler_state_dict" in checkpoint and checkpoint["scaler_state_dict"] is not None:
        scaler.load_state_dict(checkpoint["scaler_state_dict"])

    saved_epoch = checkpoint.get("epoch", 0)
    best_val_f1 = checkpoint.get("best_val_f1", checkpoint.get("val_macro_f1", 0.0))
    start_epoch = saved_epoch + 1

    print(f"ĐÃ KHÔI PHỤC CHECKPOINT: Tiếp tục từ Epoch {start_epoch} (Kỷ lục Macro-F1 trước đó: {best_val_f1:.4f})")
    return start_epoch, best_val_f1


def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: Optional[Any],
    device: torch.device,
    use_amp: bool = True,
) -> Tuple[float, float]:
    """Thực thi một epoch huấn luyện trên tập Train."""
    model.train()
    running_loss = 0.0
    correct_preds = 0
    total_samples = 0

    pbar = tqdm(dataloader, desc="Training", leave=False)
    for images, targets in pbar:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad()

        # Mixed Precision context
        if use_amp and device.type == "cuda" and scaler is not None:
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, targets)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        _, preds = torch.max(outputs, 1)
        correct_preds += torch.sum(preds == targets.data).item()
        total_samples += batch_size

        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    epoch_loss = running_loss / total_samples
    epoch_acc = correct_preds / total_samples
    return epoch_loss, epoch_acc


def validate(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    class_names: Optional[list] = None,
) -> Tuple[float, Dict[str, Any]]:
    """Thực thi đánh giá trên tập Validation."""
    model.eval()
    running_loss = 0.0
    total_samples = 0
    all_targets = []
    all_preds = []

    with torch.no_grad():
        for images, targets in tqdm(dataloader, desc="Validating", leave=False):
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, targets)

            batch_size = images.size(0)
            running_loss += loss.item() * batch_size
            total_samples += batch_size

            _, preds = torch.max(outputs, 1)
            all_targets.extend(targets.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_loss = running_loss / total_samples
    val_metrics = compute_metrics(all_targets, all_preds, class_names=class_names)
    return val_loss, val_metrics


def train_pipeline(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: Optional[Any] = None,
    num_epochs: int = 5,
    device_name: str = "cuda" if torch.cuda.is_available() else "cpu",
    output_dir: Path = Path("outputs/checkpoints"),
    run_id: Optional[str] = None,
    resume_from: Optional[Path] = None,
    use_amp: bool = True,
    class_names: Optional[list] = None,
) -> Dict[str, Any]:
    """Pipeline điều phối toàn bộ quá trình huấn luyện và lưu checkpoint."""
    device = torch.device(device_name)
    model.to(device)

    if run_id is None:
        run_id = generate_run_id()

    checkpoint_dir = output_dir / run_id
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    last_pt_path = checkpoint_dir / "last.pt"
    best_pt_path = checkpoint_dir / "best.pt"
    history_path = checkpoint_dir / "train_history.json"

    # Cấu hình AMP GradScaler
    scaler = None
    if use_amp and device.type == "cuda":
        scaler = torch.cuda.amp.GradScaler()

    start_epoch = 1
    best_val_f1 = 0.0
    history = []

    # Xử lý Resume
    if resume_from is not None and Path(resume_from).exists():
        start_epoch, best_val_f1 = load_checkpoint(
            checkpoint_path=Path(resume_from),
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            device=device_name,
        )
        if history_path.exists():
            with open(history_path, "r", encoding="utf-8") as f:
                history = json.load(f)

    print(f"=== BẮT ĐẦU HUẤN LUYỆN: Run ID = {run_id} | Thiết bị = {device} | Epochs = {start_epoch} -> {num_epochs} ===")

    for epoch in range(start_epoch, num_epochs + 1):
        t0 = time.time()
        print(f"\\n--- Epoch {epoch}/{num_epochs} ---")

        train_loss, train_acc = train_one_epoch(
            model=model,
            dataloader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            scaler=scaler,
            device=device,
            use_amp=use_amp,
        )

        val_loss, val_metrics = validate(
            model=model,
            dataloader=val_loader,
            criterion=criterion,
            device=device,
            class_names=class_names,
        )

        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_loss)
            else:
                scheduler.step()

        elapsed = time.time() - t0
        val_f1 = val_metrics["macro_f1"]
        val_acc = val_metrics["accuracy"]

        print(
            f"Epoch {epoch:02d} [{elapsed:.1f}s]: "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val Macro-F1: {val_f1:.4f}"
        )

        # Trạng thái checkpoint
        ckpt_state = {
            "epoch": epoch,
            "run_id": run_id,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict() if scheduler else None,
            "scaler_state_dict": scaler.state_dict() if scaler else None,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_macro_f1": val_f1,
            "best_val_f1": max(best_val_f1, val_f1),
        }

        # 1. Luôn lưu last.pt phục vụ resume
        save_checkpoint(ckpt_state, last_pt_path)

        # 2. Lưu best.pt khi val_macro_f1 cải thiện
        is_best = val_f1 > best_val_f1
        if is_best:
            best_val_f1 = val_f1
            ckpt_state["best_val_f1"] = best_val_f1
            save_checkpoint(ckpt_state, best_pt_path)
            print(f"⭐ KỶ LỤC MỚI: Đã cập nhật best.pt (Macro-F1 = {best_val_f1:.4f})")

        # Lưu lịch sử log
        epoch_record = {
            "epoch": epoch,
            "train_loss": round(train_loss, 4),
            "train_acc": round(train_acc, 4),
            "val_loss": round(val_loss, 4),
            "val_acc": round(val_acc, 4),
            "val_macro_f1": round(val_f1, 4),
            "is_best": is_best,
            "elapsed_seconds": round(elapsed, 2),
        }
        history.append(epoch_record)
        with open(history_path, "w", encoding="utf-8") as f:
            json.dump(history, f, indent=2, ensure_ascii=False)

    print(f"\\nHUẤN LUYỆN HOÀN TẤT. Kỷ lục Macro-F1 đạt được: {best_val_f1:.4f}")
    return {
        "run_id": run_id,
        "best_val_f1": best_val_f1,
        "last_checkpoint": str(last_pt_path),
        "best_checkpoint": str(best_pt_path),
        "history": history,
    }
''')
with open("src/utils.py", "w", encoding="utf-8") as f: f.write('''"""Module utils.py

Trách nhiệm:
- Cung cấp các hàm tiện ích dùng chung trong toàn bộ dự án.
"""

import os
import random
import numpy as np
import torch


def set_seed(seed: int = 42) -> None:
    """Thiết lập seed ngẫu nhiên cho các thư viện để đảm bảo khả năng tái lập kết quả thực nghiệm.

    Args:
        seed (int): Giá trị seed cần thiết lập (mặc định là 42).
    """
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
''')

# Viết file splits/split_v1.csv
with open("splits/split_v1.csv", "w", encoding="utf-8") as f:
    f.write('''image_id,label,rel_image_path,rel_mask_path,has_mask,group_id,split,file_hash,phash
COVID-1.png,COVID,COVID/images/COVID-1.png,COVID/masks/COVID-1.png,True,G00000,test,f562f6a6290c0a8e27afc9f994625380,d17b561a59191b1e
COVID-10.png,COVID,COVID/images/COVID-10.png,COVID/masks/COVID-10.png,True,G00001,test,f29158352a454f1d39ed9924406f99ec,986fc65e4b595c41
COVID-100.png,COVID,COVID/images/COVID-100.png,COVID/masks/COVID-100.png,True,G00002,train,6da68dc77f90d6ed08fc41bf3c3d3cae,992a061a4f79794f
COVID-1000.png,COVID,COVID/images/COVID-1000.png,COVID/masks/COVID-1000.png,True,G00003,train,641cebe2e5e95cfcd94a6dc2b5d1b3ee,91ea861e595f1a59
COVID-1001.png,COVID,COVID/images/COVID-1001.png,COVID/masks/COVID-1001.png,True,G00004,train,b34c634b7eb9d9fd6b801b5218d2e684,c44f0b0c66b7b1d3
COVID-1002.png,COVID,COVID/images/COVID-1002.png,COVID/masks/COVID-1002.png,True,G00005,train,422f0565e6f9170471119772d1c04908,b82d8517c31f1e47
COVID-1003.png,COVID,COVID/images/COVID-1003.png,COVID/masks/COVID-1003.png,True,G00006,train,41550c0344c667f07122a3d8d766f9dc,987b05161a475f7a
COVID-1004.png,COVID,COVID/images/COVID-1004.png,COVID/masks/COVID-1004.png,True,G00007,train,20b6575c31fc2a456335aeab1ba6e034,817b085e187b79e9
COVID-1005.png,COVID,COVID/images/COVID-1005.png,COVID/masks/COVID-1005.png,True,G00008,train,711e4fb3cb56e2782c92375b29361621,f969c696545639c1
COVID-1006.png,COVID,COVID/images/COVID-1006.png,COVID/masks/COVID-1006.png,True,G00009,val,2a2482e5c9d19e819c2890f53abbca07,997f06184d194f4f
COVID-1007.png,COVID,COVID/images/COVID-1007.png,COVID/masks/COVID-1007.png,True,G00010,train,b437244040aaf8ece7c3acd778011715,895f065a4dede8a1
COVID-1008.png,COVID,COVID/images/COVID-1008.png,COVID/masks/COVID-1008.png,True,G00011,train,d943b445e77b1322989f67980e4c3442,9d6c409753911dd7
COVID-1009.png,COVID,COVID/images/COVID-1009.png,COVID/masks/COVID-1009.png,True,G00012,train,f29444e7785c1c5e855bd8f14100d579,912a86961a973f9d
COVID-101.png,COVID,COVID/images/COVID-101.png,COVID/masks/COVID-101.png,True,G00013,train,9863c004407e2c99e62c9502dd1848b4,997b87161e491d1b
COVID-1010.png,COVID,COVID/images/COVID-1010.png,COVID/masks/COVID-1010.png,True,G00014,val,eda7c8d33a7b248b5aed2ffe7da9f0cc,913b44061f195f5f
COVID-1011.png,COVID,COVID/images/COVID-1011.png,COVID/masks/COVID-1011.png,True,G00015,val,1f6cea7496fed4e1aa8f93f7eb9ae7c5,dc2ef04309d35a5d
COVID-1012.png,COVID,COVID/images/COVID-1012.png,COVID/masks/COVID-1012.png,True,G00016,test,3894c52f655531a9a903c27f3d7e225d,d42ec15f894d1d56
COVID-1013.png,COVID,COVID/images/COVID-1013.png,COVID/masks/COVID-1013.png,True,G00017,test,64a5bbe062c9ccd69a3b854d92f92ca8,d13f064a5b575b09
COVID-1014.png,COVID,COVID/images/COVID-1014.png,COVID/masks/COVID-1014.png,True,G00018,train,a5ac932c1215dcb6261aa07f58fb9e0a,dd7c431a1d49554e
COVID-1015.png,COVID,COVID/images/COVID-1015.png,COVID/masks/COVID-1015.png,True,G00019,test,9978b12e04aa3c12de5d9a28f20a83af,837c9d716a7a6868
COVID-1016.png,COVID,COVID/images/COVID-1016.png,COVID/masks/COVID-1016.png,True,G00020,train,f2d56e2515cc1f3ec30b9c3c4cda9953,d42f53434f4f4a0e
COVID-1017.png,COVID,COVID/images/COVID-1017.png,COVID/masks/COVID-1017.png,True,G00021,train,ae32ccc653e4e84308d8e05a810dc7e2,996d461e21596b5e
COVID-1018.png,COVID,COVID/images/COVID-1018.png,COVID/masks/COVID-1018.png,True,G00022,test,f441ea861f677529d1c06fe6345c50f6,d42fc3535c4e0b1d
COVID-1019.png,COVID,COVID/images/COVID-1019.png,COVID/masks/COVID-1019.png,True,G00023,train,0440accd4567c96ffbc439b22a07005e,995f041e185b79f8
COVID-102.png,COVID,COVID/images/COVID-102.png,COVID/masks/COVID-102.png,True,G00024,train,da18441924afa55afeb2e320e6504e29,992ae7007f69c03f
COVID-1020.png,COVID,COVID/images/COVID-1020.png,COVID/masks/COVID-1020.png,True,G00025,train,0b7a97c505f1f8250be9e0914221ec9a,d979c55616c4165e
COVID-1021.png,COVID,COVID/images/COVID-1021.png,COVID/masks/COVID-1021.png,True,G00026,train,3d016c3a3d079a00f6bb190299644a9d,d56d579e1a1c0e1a
COVID-1022.png,COVID,COVID/images/COVID-1022.png,COVID/masks/COVID-1022.png,True,G00027,train,f364100e0cfd1dd0c2ce20fd03cf1527,917b57065e5c4b13
COVID-1023.png,COVID,COVID/images/COVID-1023.png,COVID/masks/COVID-1023.png,True,G00028,train,5f5bbb6f85ad26de1d4f03b3763660b9,b9394716164b1d5d
COVID-1024.png,COVID,COVID/images/COVID-1024.png,COVID/masks/COVID-1024.png,True,G00029,train,1b1d40da3e648452a4c6cf172accd588,dc2d434f2e303cbc
COVID-1025.png,COVID,COVID/images/COVID-1025.png,COVID/masks/COVID-1025.png,True,G00030,train,1e5199f2c06c5c131309f8d2e67ed63e,dd0d571a1e4d4b19
COVID-1026.png,COVID,COVID/images/COVID-1026.png,COVID/masks/COVID-1026.png,True,G00031,train,fc02f8a1721f67c94c65f1f9c0c1d118,bc4d47465f591c0e
COVID-1027.png,COVID,COVID/images/COVID-1027.png,COVID/masks/COVID-1027.png,True,G00032,val,5ece33729cf3c03f30b8d86f11fb80db,913b96165e4f1b19
COVID-1028.png,COVID,COVID/images/COVID-1028.png,COVID/masks/COVID-1028.png,True,G00033,train,0ffd904f05cb38ebeb74d40977cf88ae,892f063e3a7b7960
COVID-1029.png,COVID,COVID/images/COVID-1029.png,COVID/masks/COVID-1029.png,True,G00034,train,3b34760295806af4ab977a1c73460035,9c4d43075f490f4f
COVID-103.png,COVID,COVID/images/COVID-103.png,COVID/masks/COVID-103.png,True,G00035,val,4f054f061e219031d3abbd9b7db11303,9d2c465b1b4f1a4d
COVID-1030.png,COVID,COVID/images/COVID-1030.png,COVID/masks/COVID-1030.png,True,G00036,train,ab90fc6e323d159d203de0e5e8bb7806,913b471a4f4f1a1b
COVID-1031.png,COVID,COVID/images/COVID-1031.png,COVID/masks/COVID-1031.png,True,G00037,train,0585d2567cb15521a2160828926f7ed3,886f835b49657c78
COVID-1032.png,COVID,COVID/images/COVID-1032.png,COVID/masks/COVID-1032.png,True,G00038,test,6e50783d746241708e12f0d02d644f88,99394756165a5a5e
COVID-1033.png,COVID,COVID/images/COVID-1033.png,COVID/masks/COVID-1033.png,True,G00039,train,99638756f0f3ca8866372046319610ac,817a465a1a5f5d39
COVID-1034.png,COVID,COVID/images/COVID-1034.png,COVID/masks/COVID-1034.png,True,G00040,train,7003b2fb37b33eff77c666236dd8dfc7,986fc71e1a415b4b
COVID-1035.png,COVID,COVID/images/COVID-1035.png,COVID/masks/COVID-1035.png,True,G00041,val,c85c8d75cf5673cefeec3ee2121e601a,897f0e18195d5753
COVID-1036.png,COVID,COVID/images/COVID-1036.png,COVID/masks/COVID-1036.png,True,G00042,train,8f6d679cb91c92da07706ae6e7c566f2,8978061a5a7d795d
COVID-1037.png,COVID,COVID/images/COVID-1037.png,COVID/masks/COVID-1037.png,True,G00043,train,5f5b53a6d93f76714bd556bf786e6a11,d11b131f393931b3
COVID-1038.png,COVID,COVID/images/COVID-1038.png,COVID/masks/COVID-1038.png,True,G00044,test,8489cc060ca2a30cc3703b9809e4e5d3,91f86e867a857e18
COVID-1039.png,COVID,COVID/images/COVID-1039.png,COVID/masks/COVID-1039.png,True,G00045,train,e28bd145f0525a276eeb87c11e754120,dd2ec31f0d061d4d
COVID-104.png,COVID,COVID/images/COVID-104.png,COVID/masks/COVID-104.png,True,G00046,train,aeb7412ef18fdd0b092aae10a1cb6e8d,d879c7161e5a1a5a
COVID-1040.png,COVID,COVID/images/COVID-1040.png,COVID/masks/COVID-1040.png,True,G00047,train,ecaa72b0546e97bb9f59127d4e40315d,980d061e4f5a4f5f
COVID-1041.png,COVID,COVID/images/COVID-1041.png,COVID/masks/COVID-1041.png,True,G00048,train,f504d7210018a5a350a968246f5536c0,9b780f1a5ad869e1
COVID-1042.png,COVID,COVID/images/COVID-1042.png,COVID/masks/COVID-1042.png,True,G00049,val,8eeab2985a667f6fb11442323cf6384f,9979c7161e594959
COVID-1043.png,COVID,COVID/images/COVID-1043.png,COVID/masks/COVID-1043.png,True,G00050,train,57fe93f9a07c4e6b0254eb3a3cffc60e,d959561e18393da5
COVID-1044.png,COVID,COVID/images/COVID-1044.png,COVID/masks/COVID-1044.png,True,G00051,train,03bde6766e6a5429d9c792683829fe80,d46f025b4f081b5f
COVID-1045.png,COVID,COVID/images/COVID-1045.png,COVID/masks/COVID-1045.png,True,G00052,train,c774f3e94c420870d5937a26ede1379f,956f020b0f5f0f0f
COVID-1046.png,COVID,COVID/images/COVID-1046.png,COVID/masks/COVID-1046.png,True,G00053,train,fddafead84db457677f26c59afb89853,cc4c471f1f486de4
COVID-1047.png,COVID,COVID/images/COVID-1047.png,COVID/masks/COVID-1047.png,True,G00054,train,b8fa4d8a7373c5d19a9d591a4e01a514,9d68861a4d5d394f
COVID-1048.png,COVID,COVID/images/COVID-1048.png,COVID/masks/COVID-1048.png,True,G00055,val,fc5a9c87f6bbc6a7a82f2e16a669831f,dd2c430b5f4b0f0d
COVID-1049.png,COVID,COVID/images/COVID-1049.png,COVID/masks/COVID-1049.png,True,G00056,val,d475d9b379311355989defa0573d6a3d,cd4e44161e5a7a7c
COVID-105.png,COVID,COVID/images/COVID-105.png,COVID/masks/COVID-105.png,True,G00057,test,2f71ca55a35afee7c7107406943f8e64,9d20d20aef2f815f
COVID-1050.png,COVID,COVID/images/COVID-1050.png,COVID/masks/COVID-1050.png,True,G00058,test,69de877f6a11a365249888efb4b95ccf,c56ad2584a5f584f
COVID-1051.png,COVID,COVID/images/COVID-1051.png,COVID/masks/COVID-1051.png,True,G00059,train,c532b87701be2044072e45feb4cb2c67,d4bc42875a936bca
COVID-1052.png,COVID,COVID/images/COVID-1052.png,COVID/masks/COVID-1052.png,True,G00060,train,5825db2e8b174c3aebed918c8192d62d,916bd65a4b585879
COVID-1053.png,COVID,COVID/images/COVID-1053.png,COVID/masks/COVID-1053.png,True,G00061,test,23828496f8808117cc852892543afa52,cc26130f462fc76e
COVID-1054.png,COVID,COVID/images/COVID-1054.png,COVID/masks/COVID-1054.png,True,G00062,train,60bcda139331ef771fbce0973c6180f5,8c65f21ac11fcf49
COVID-1055.png,COVID,COVID/images/COVID-1055.png,COVID/masks/COVID-1055.png,True,G00063,train,574bcad72ebf059355378a4d9c594016,8c0d071e5f4d7c6c
COVID-1056.png,COVID,COVID/images/COVID-1056.png,COVID/masks/COVID-1056.png,True,G00064,test,4777d27dbc9a70eb816ae262cbeb8ee6,916f1618695acf1e
COVID-1057.png,COVID,COVID/images/COVID-1057.png,COVID/masks/COVID-1057.png,True,G00065,train,5da69b241b0df94695cb13b81c146560,9129c628a9f97979
COVID-1058.png,COVID,COVID/images/COVID-1058.png,COVID/masks/COVID-1058.png,True,G00066,train,5a258309e2db70fa5efcbd682a5874a2,c54e430b4d4b6db6
COVID-1059.png,COVID,COVID/images/COVID-1059.png,COVID/masks/COVID-1059.png,True,G00067,train,d31e0d0174e85185e73fec436db22901,9d7d071a194d5a5c
COVID-106.png,COVID,COVID/images/COVID-106.png,COVID/masks/COVID-106.png,True,G00068,train,112d8695e04360a6ae356a95b6ab2214,dd2ec21f544d1c47
COVID-1060.png,COVID,COVID/images/COVID-1060.png,COVID/masks/COVID-1060.png,True,G00069,test,83f0c050d1079c87c58cf1913308e498,c135c269786f3939
COVID-1061.png,COVID,COVID/images/COVID-1061.png,COVID/masks/COVID-1061.png,True,G00070,val,4d8844b1e01521ebb56668a3620dd606,db0b0c1d35667a71
COVID-1062.png,COVID,COVID/images/COVID-1062.png,COVID/masks/COVID-1062.png,True,G00071,val,5f8301dfc1fdd0bb4f59cfd43e87bbe2,d02967161e1e137f
COVID-1063.png,COVID,COVID/images/COVID-1063.png,COVID/masks/COVID-1063.png,True,G00072,train,8bcbff2e3dc9b3b870e1f1b185b8b84a,99a9aeaeba9a0e18
COVID-1064.png,COVID,COVID/images/COVID-1064.png,COVID/masks/COVID-1064.png,True,G00073,train,868c8b02a17fa6e9bf201e4e880d91ff,d17f54143a1a7f12
COVID-1065.png,COVID,COVID/images/COVID-1065.png,COVID/masks/COVID-1065.png,True,G00074,val,86b1e3561dd37524d8807408350fff89,d62fd5470e060f4e
COVID-1066.png,COVID,COVID/images/COVID-1066.png,COVID/masks/COVID-1066.png,True,G00075,train,76037e4b3fdce076d97b9c14f90dfb0b,b82dc496c71b1f49
COVID-1067.png,COVID,COVID/images/COVID-1067.png,COVID/masks/COVID-1067.png,True,G00076,train,1ed4803fe8c8f810b81c4f087a35c89b,901b1a1a7be9e4e6
COVID-1068.png,COVID,COVID/images/COVID-1068.png,COVID/masks/COVID-1068.png,True,G00077,test,d893c11f9211c5e8eb9433f6ff82c32b,917f861a057d784d
COVID-1069.png,COVID,COVID/images/COVID-1069.png,COVID/masks/COVID-1069.png,True,G00078,train,1b772c5e32dac333535ad0bba6080868,9c61e1166b5d1d59
COVID-107.png,COVID,COVID/images/COVID-107.png,COVID/masks/COVID-107.png,True,G00079,train,db08feee7df0fc3bccfdc51be848f7fa,c53e27937ac17ac0
COVID-1070.png,COVID,COVID/images/COVID-1070.png,COVID/masks/COVID-1070.png,True,G00080,train,e3462978bc7844c3b6e2e339512fc231,d13b57065a1a5a97
COVID-1071.png,COVID,COVID/images/COVID-1071.png,COVID/masks/COVID-1071.png,True,G00081,train,031d8f26ccc394e326d64f91c11d0ee1,897dd61a44531b4f
COVID-1072.png,COVID,COVID/images/COVID-1072.png,COVID/masks/COVID-1072.png,True,G00082,train,cc325ceed9c656f7a4b7f9da5d42f1f7,9a69e596c706b64a
COVID-1073.png,COVID,COVID/images/COVID-1073.png,COVID/masks/COVID-1073.png,True,G00083,val,2a90b2906ac09e244f1a2a1b6a7d911b,d11b161e597979a4
COVID-1074.png,COVID,COVID/images/COVID-1074.png,COVID/masks/COVID-1074.png,True,G00084,train,fb6cf32849fbd8f84a5edbdd99cb924c,817d16185f0e1b5f
COVID-1075.png,COVID,COVID/images/COVID-1075.png,COVID/masks/COVID-1075.png,True,G00085,train,c7a1d6838363abf5fe993a9f7f79eda6,917916061a1b5f5f
COVID-1076.png,COVID,COVID/images/COVID-1076.png,COVID/masks/COVID-1076.png,True,G00086,train,8a801ac4df1ed73d85063599bb9e5fbd,986de492e19ec31e
COVID-1077.png,COVID,COVID/images/COVID-1077.png,COVID/masks/COVID-1077.png,True,G00087,train,3a0286ca32f42b486da258e151b44380,817f860a1d4f1d5d
COVID-1078.png,COVID,COVID/images/COVID-1078.png,COVID/masks/COVID-1078.png,True,G00088,train,6f6b202143ddcdc1d1a6847314750b07,8d65c21b431f4f4d
COVID-1079.png,COVID,COVID/images/COVID-1079.png,COVID/masks/COVID-1079.png,True,G00089,train,10da2f32915318886e6e99736e36345f,d869e59652595e68
COVID-108.png,COVID,COVID/images/COVID-108.png,COVID/masks/COVID-108.png,True,G00090,train,11541d02fbbbf129ff00646457cc7894,816e1e5945e6f2a6
COVID-1080.png,COVID,COVID/images/COVID-1080.png,COVID/masks/COVID-1080.png,True,G00091,train,b5972c74865ae6000db53bd38ae3c6ca,817c165a5a4f7959
COVID-1081.png,COVID,COVID/images/COVID-1081.png,COVID/masks/COVID-1081.png,True,G00092,train,40ad0a419f1831adbf60b9a34e002de6,ca742e4f3d3c2c2c
COVID-1082.png,COVID,COVID/images/COVID-1082.png,COVID/masks/COVID-1082.png,True,G00093,val,b33c65ed8b6880f8b22365704d4afced,8c68849acb9c9e9f
COVID-1083.png,COVID,COVID/images/COVID-1083.png,COVID/masks/COVID-1083.png,True,G00094,test,ab5e260348cef86843c3664fa36bc466,817d065a1f491f4f
COVID-1084.png,COVID,COVID/images/COVID-1084.png,COVID/masks/COVID-1084.png,True,G00095,val,7e31361ecad80a70bb3c928c7f1381ee,d9298716171e5c4f
COVID-1085.png,COVID,COVID/images/COVID-1085.png,COVID/masks/COVID-1085.png,True,G00096,test,7fe8510625d4c75fdeeac9722b2272cb,9c69e296c31d951d
COVID-1086.png,COVID,COVID/images/COVID-1086.png,COVID/masks/COVID-1086.png,True,G00097,train,ff0984129ca85de36ec20863ca7b243d,d979c7161e045f19
COVID-1087.png,COVID,COVID/images/COVID-1087.png,COVID/masks/COVID-1087.png,True,G00098,train,24e079e47f2ff3f6b8d047fefb23c414,f969c6165e461a6a
COVID-1088.png,COVID,COVID/images/COVID-1088.png,COVID/masks/COVID-1088.png,True,G00099,train,09b276ab39ce7598ab40440f0881e1e3,9073e79eca1a5e50
COVID-1089.png,COVID,COVID/images/COVID-1089.png,COVID/masks/COVID-1089.png,True,G00100,test,b92b2ab2c9d014cdcd510895705d3c8f,817b07961ecf1ed0
COVID-109.png,COVID,COVID/images/COVID-109.png,COVID/masks/COVID-109.png,True,G00101,train,0bc92192ffd6b2f7299ba8774c3196ea,dd2fd61e471a0945
COVID-1090.png,COVID,COVID/images/COVID-1090.png,COVID/masks/COVID-1090.png,True,G00102,train,0dbd95f8d23d7e307d39f21465079281,dc2b531d28f0c793
COVID-1091.png,COVID,COVID/images/COVID-1091.png,COVID/masks/COVID-1091.png,True,G00103,train,85105e98d54408af74da55fbd986c737,983f05d4c2530f5f
COVID-1092.png,COVID,COVID/images/COVID-1092.png,COVID/masks/COVID-1092.png,True,G00104,val,cae9f22eae7d0f7b7f39529db8eac529,882f430a6d6f78ec
COVID-1093.png,COVID,COVID/images/COVID-1093.png,COVID/masks/COVID-1093.png,True,G00105,test,7f60ae0e111f6e58869ae10ece573f08,9c65d01fc1585f4b
COVID-1094.png,COVID,COVID/images/COVID-1094.png,COVID/masks/COVID-1094.png,True,G00106,train,10e847e143522143239ae2d884e700f2,9c69e594c38b1cc7
COVID-1095.png,COVID,COVID/images/COVID-1095.png,COVID/masks/COVID-1095.png,True,G00107,train,be6f59bac6445abba965e1d66c21ee1a,d42f870e0c4c6fce
COVID-1096.png,COVID,COVID/images/COVID-1096.png,COVID/masks/COVID-1096.png,True,G00108,train,2646d7593575d43a7a8535ac54149da9,9129c62929f97979
COVID-1097.png,COVID,COVID/images/COVID-1097.png,COVID/masks/COVID-1097.png,True,G00109,train,bee49db3088c8b2c6cf8bca4b45eede1,f979c6c6161e5849
COVID-1098.png,COVID,COVID/images/COVID-1098.png,COVID/masks/COVID-1098.png,True,G00110,train,30aee2e61b2f460fde11f32c436389d8,d42ea3830fc62797
COVID-1099.png,COVID,COVID/images/COVID-1099.png,COVID/masks/COVID-1099.png,True,G00111,train,a6573f9c1ec18a000fc24480e9367849,982d8097834f1fd7
COVID-11.png,COVID,COVID/images/COVID-11.png,COVID/masks/COVID-11.png,True,G00112,train,f487c8a6116b7099336edd0d52821baa,9c3c420f7f483d74
COVID-110.png,COVID,COVID/images/COVID-110.png,COVID/masks/COVID-110.png,True,G00113,train,9b1c4cd3ceaf30ae1dc3db619f332258,cc2a434c6a6ef2ba
COVID-1100.png,COVID,COVID/images/COVID-1100.png,COVID/masks/COVID-1100.png,True,G00114,train,57bc5ea485e7dee7c1ccb71673f11e79,f92d861603533e5b
COVID-1101.png,COVID,COVID/images/COVID-1101.png,COVID/masks/COVID-1101.png,True,G00115,train,74df7dc24241d552b784a03b33158a6c,d52e83870cc56edc
COVID-1102.png,COVID,COVID/images/COVID-1102.png,COVID/masks/COVID-1102.png,True,G00116,val,a652a3ecc9b1e64ba43ef0b1953ebe92,d979851606471e5f
COVID-1103.png,COVID,COVID/images/COVID-1103.png,COVID/masks/COVID-1103.png,True,G00117,test,8fa9de678fc2e1da46fe86c002d0c57f,c426505bd95d693d
COVID-1104.png,COVID,COVID/images/COVID-1104.png,COVID/masks/COVID-1104.png,True,G00118,train,15585406d33f1abb877705b58c8d335e,d42f02474b3c6ced
COVID-1105.png,COVID,COVID/images/COVID-1105.png,COVID/masks/COVID-1105.png,True,G00119,val,ac0579b9b2fade3405d0b06a8f1b085a,9d6dc3021b431b5f
COVID-1106.png,COVID,COVID/images/COVID-1106.png,COVID/masks/COVID-1106.png,True,G00120,test,ae1f5192a1b932d85cde01b99fa250b8,942601531b1f7d6f
COVID-1107.png,COVID,COVID/images/COVID-1107.png,COVID/masks/COVID-1107.png,True,G00121,train,cc0367594b61650ee5939d45ed72388b,89798686188d3f9f
COVID-1108.png,COVID,COVID/images/COVID-1108.png,COVID/masks/COVID-1108.png,True,G00122,train,1d1a43ed6bdc0f833a33a3f93276c7ac,d87bc5164e4b1a5a
COVID-1109.png,COVID,COVID/images/COVID-1109.png,COVID/masks/COVID-1109.png,True,G00123,train,21afe8c77a731bcd72a4a230c96edbf4,8e24625b0b4b6f4f
COVID-111.png,COVID,COVID/images/COVID-111.png,COVID/masks/COVID-111.png,True,G00124,train,5f21a41c8538df777d8e536f7ad21ca2,847e435a1f4b1d4d
COVID-1110.png,COVID,COVID/images/COVID-1110.png,COVID/masks/COVID-1110.png,True,G00125,train,964f51e136ec692cf3e2ed0060ae05c5,85680e805f57df17
COVID-1111.png,COVID,COVID/images/COVID-1111.png,COVID/masks/COVID-1111.png,True,G00126,train,8ee941404ac3e5f937b3d4b11ddd4a36,9c65c61ec15a4f5a
COVID-1112.png,COVID,COVID/images/COVID-1112.png,COVID/masks/COVID-1112.png,True,G00127,train,d52675182af7fb7ed518b7b50638252d,9969c5161fc71b1a
COVID-1113.png,COVID,COVID/images/COVID-1113.png,COVID/masks/COVID-1113.png,True,G00128,train,bf3695d53d6e58f47b063234332bc788,8168165b5a5f785b
COVID-1114.png,COVID,COVID/images/COVID-1114.png,COVID/masks/COVID-1114.png,True,G00129,val,8643b3fca5be14097ccccbd9ffb5de78,d938d51a17c05b5d
COVID-1115.png,COVID,COVID/images/COVID-1115.png,COVID/masks/COVID-1115.png,True,G00130,train,62cfe387a4650bfc8bb0550bf2edd431,cf6d1012eb490f6b
COVID-1116.png,COVID,COVID/images/COVID-1116.png,COVID/masks/COVID-1116.png,True,G00131,train,5e198b4f2bc50b3c0d87ab0dbbed3f83,cc2d4b0a2eaa2f8f
COVID-1117.png,COVID,COVID/images/COVID-1117.png,COVID/masks/COVID-1117.png,True,G00132,train,5cb954c5009a641988973328333e2ffa,c135c268796f3939
COVID-1118.png,COVID,COVID/images/COVID-1118.png,COVID/masks/COVID-1118.png,True,G00133,val,f3110a82f2ad0c6df0187423b713355a,dc69d416435c5f49
COVID-1119.png,COVID,COVID/images/COVID-1119.png,COVID/masks/COVID-1119.png,True,G00134,test,b8d04969750227114fc292e326b9e118,c83cd7531d4e5259
COVID-112.png,COVID,COVID/images/COVID-112.png,COVID/masks/COVID-112.png,True,G00135,train,b66db83b183446e4153e1972f9992b44,d869c5165b595e49
COVID-1120.png,COVID,COVID/images/COVID-1120.png,COVID/masks/COVID-1120.png,True,G00136,test,c93bf82d4e6a4efe040cc31cdd17f086,99a5a7af399b0e08
COVID-1121.png,COVID,COVID/images/COVID-1121.png,COVID/masks/COVID-1121.png,True,G00137,train,2e8f30bb33e011ca2acd4a4ec36553c7,b92d051616531f5f
COVID-1122.png,COVID,COVID/images/COVID-1122.png,COVID/masks/COVID-1122.png,True,G00138,train,35a6c7fd5f9c150dd04b921830d73ad1,c85f12194d0abfa7
COVID-1123.png,COVID,COVID/images/COVID-1123.png,COVID/masks/COVID-1123.png,True,G00139,train,4c4a7ae5607059587d0912057f5d3f58,dd28d3c35f55420b
COVID-1124.png,COVID,COVID/images/COVID-1124.png,COVID/masks/COVID-1124.png,True,G00140,val,1a75d350c73a0c04813bc58f85c1fa22,bc2de49693431c1f
COVID-1125.png,COVID,COVID/images/COVID-1125.png,COVID/masks/COVID-1125.png,True,G00141,train,b7e40a76e680e8b7800595210a2c1e64,d62dc20d2e2e2f66
COVID-1126.png,COVID,COVID/images/COVID-1126.png,COVID/masks/COVID-1126.png,True,G00142,train,33278551d9c87843f1a6d79857ee20c3,987d8716174f1707
COVID-1127.png,COVID,COVID/images/COVID-1127.png,COVID/masks/COVID-1127.png,True,G00143,val,9b8428237765159299d50dd28063b9d2,ac24d21b431f574f
COVID-1128.png,COVID,COVID/images/COVID-1128.png,COVID/masks/COVID-1128.png,True,G00144,train,a0fdbd95c2a91048a9b710727bfd29bc,dd6ac2535f40594d
COVID-1129.png,COVID,COVID/images/COVID-1129.png,COVID/masks/COVID-1129.png,True,G00145,test,6743ef034572faa373593518b09187ac,c47e27c33bb468b0
COVID-113.png,COVID,COVID/images/COVID-113.png,COVID/masks/COVID-113.png,True,G00146,test,5b6b3fe6c1c0f0df61795c2f87d579d1,817b5618191bbb3d
COVID-1130.png,COVID,COVID/images/COVID-1130.png,COVID/masks/COVID-1130.png,True,G00147,train,c9156c835d837ae58b0d5fc82a33f7e4,957c870a87471e97
COVID-1131.png,COVID,COVID/images/COVID-1131.png,COVID/masks/COVID-1131.png,True,G00148,train,47d825580327bfa578fc54a8cbf4a475,9d6cd213471f514b
COVID-1132.png,COVID,COVID/images/COVID-1132.png,COVID/masks/COVID-1132.png,True,G00149,train,68e469b1220f8b0e9b3d81b843272db5,847f239279e17692
COVID-1133.png,COVID,COVID/images/COVID-1133.png,COVID/masks/COVID-1133.png,True,G00150,train,dd7623613c77eb779dc69ece6fbec698,8828e056b0e73ffc
COVID-1134.png,COVID,COVID/images/COVID-1134.png,COVID/masks/COVID-1134.png,True,G00151,train,feb23c78146a86a0333b0eccf1d80f5d,93780c5b5bd0e2cd
COVID-1135.png,COVID,COVID/images/COVID-1135.png,COVID/masks/COVID-1135.png,True,G00152,train,e308de36726b904d2a6ed188c4323d34,dc6c535f0d1c0f2a
COVID-1136.png,COVID,COVID/images/COVID-1136.png,COVID/masks/COVID-1136.png,True,G00153,train,31ccf8c346845f4fb0c60cb7b9192f59,9915461e4d1e5e4f
COVID-1137.png,COVID,COVID/images/COVID-1137.png,COVID/masks/COVID-1137.png,True,G00154,train,df738687873ebe522bce53e2a3231c4b,d63d0b262c3166f6
COVID-1138.png,COVID,COVID/images/COVID-1138.png,COVID/masks/COVID-1138.png,True,G00155,train,2cbcaa114ba3c154c99654a0b5e08bce,987fd71a5c435a18
COVID-1139.png,COVID,COVID/images/COVID-1139.png,COVID/masks/COVID-1139.png,True,G00156,val,1564768efeb7cd1ed603bf172f79ab24,9d64c21f45571e4d
COVID-114.png,COVID,COVID/images/COVID-114.png,COVID/masks/COVID-114.png,True,G00157,val,4a44994f4bc7af9c2f3968d51eb45616,d42aa70a5d5d193d
COVID-1140.png,COVID,COVID/images/COVID-1140.png,COVID/masks/COVID-1140.png,True,G00158,test,6ddc7cbcfcc02225af0870945bcb3aad,812bd659531a795e
COVID-1141.png,COVID,COVID/images/COVID-1141.png,COVID/masks/COVID-1141.png,True,G00159,test,675ec43a1d289b7232a806946423da1d,bc3d434b1b451e1e
COVID-1142.png,COVID,COVID/images/COVID-1142.png,COVID/masks/COVID-1142.png,True,G00160,train,06c32ed2e72ed0247cf47048b1e8ab1f,dd2cf0425b53615e
COVID-1143.png,COVID,COVID/images/COVID-1143.png,COVID/masks/COVID-1143.png,True,G00161,train,e257855a645edc3329662a82abecd850,d43f831b65461c5d
COVID-1144.png,COVID,COVID/images/COVID-1144.png,COVID/masks/COVID-1144.png,True,G00162,train,caf56fdea69de2162588719012d3c9ee,9d2ce096c37a2f68
COVID-1145.png,COVID,COVID/images/COVID-1145.png,COVID/masks/COVID-1145.png,True,G00163,test,d8f0ee96c5f0d2599a9cd277a466df21,853d1b32367526f2
COVID-1146.png,COVID,COVID/images/COVID-1146.png,COVID/masks/COVID-1146.png,True,G00164,train,7f36205a51eacd82f8b4b10e3a81382c,942c574b0f1d5f49
COVID-1147.png,COVID,COVID/images/COVID-1147.png,COVID/masks/COVID-1147.png,True,G00165,train,787d54b8634d9e4103f0fa83a2fbf611,d1ffc25a544a1a6c
COVID-1148.png,COVID,COVID/images/COVID-1148.png,COVID/masks/COVID-1148.png,True,G00166,train,967a879815d2321348091a637e0641c9,c14d0e3e1f4e2d3c
COVID-1149.png,COVID,COVID/images/COVID-1149.png,COVID/masks/COVID-1149.png,True,G00167,train,b9983782a95227df887c9e0a1196bca7,fd66831f740e4649
COVID-115.png,COVID,COVID/images/COVID-115.png,COVID/masks/COVID-115.png,True,G00108,train,6f19fd53121bace7f0f9422fae02220e,9129c62929f97979
COVID-1150.png,COVID,COVID/images/COVID-1150.png,COVID/masks/COVID-1150.png,True,G00168,train,0f941e2fd49b4ec84daca6869b45bdba,dc2ef04303d35a5d
COVID-1151.png,COVID,COVID/images/COVID-1151.png,COVID/masks/COVID-1151.png,True,G00169,train,b877971defd62475077054280f74d12f,9d285742175b5b4b
COVID-1152.png,COVID,COVID/images/COVID-1152.png,COVID/masks/COVID-1152.png,True,G00170,train,df441a5e1f6f95e59255cb148ad6566b,9426030f4f476e6f
COVID-1153.png,COVID,COVID/images/COVID-1153.png,COVID/masks/COVID-1153.png,True,G00171,train,1d29556d703095198f72ebac69c65f20,dd7c421a1d495d4e
COVID-1154.png,COVID,COVID/images/COVID-1154.png,COVID/masks/COVID-1154.png,True,G00172,train,b76c968e9b3347e3235d83c1b5faa594,8c4d43431b5b5d4f
COVID-1155.png,COVID,COVID/images/COVID-1155.png,COVID/masks/COVID-1155.png,True,G00173,val,c1e697205870da62fd595bfe669f8803,9c6ac30f4f0e0e2f
COVID-1156.png,COVID,COVID/images/COVID-1156.png,COVID/masks/COVID-1156.png,True,G00174,train,32e8aaed19f319c7ae9513b2598b12c3,996d463e2159695e
COVID-1157.png,COVID,COVID/images/COVID-1157.png,COVID/masks/COVID-1157.png,True,G00175,train,f376b79fdd04e9d98282f0ed40cc6a2e,9d4d43031b131f6f
COVID-1158.png,COVID,COVID/images/COVID-1158.png,COVID/masks/COVID-1158.png,True,G00176,test,50eae0eeb484a139fe984e5680a5f1c2,d929c6121a5f5b5a
COVID-1159.png,COVID,COVID/images/COVID-1159.png,COVID/masks/COVID-1159.png,True,G00177,val,f30cca9e6fcef2d81a08f8dbf28b2152,8d75d21a671e1b45
COVID-116.png,COVID,COVID/images/COVID-116.png,COVID/masks/COVID-116.png,True,G00178,train,01d0a7a57b3f1bd2545544d4cdebe08b,9179561e592ba8b5
COVID-1160.png,COVID,COVID/images/COVID-1160.png,COVID/masks/COVID-1160.png,True,G00179,val,9c8a1befc87b5442d83d70361fc98ca8,9959461e5e195a5b
COVID-1161.png,COVID,COVID/images/COVID-1161.png,COVID/masks/COVID-1161.png,True,G00180,test,b304bf9e01a159738aa1c7bd6fe91e60,dd2a535d444a9ba3
COVID-1162.png,COVID,COVID/images/COVID-1162.png,COVID/masks/COVID-1162.png,True,G00181,train,67a015c107796d96256dfbd72fa5fb31,dd2cd25be21af04c
COVID-1163.png,COVID,COVID/images/COVID-1163.png,COVID/masks/COVID-1163.png,True,G00182,train,7a7f6dbd2e6eb05f2ca23ceb1994dfbc,9c6ce10f1f690f1c
COVID-1164.png,COVID,COVID/images/COVID-1164.png,COVID/masks/COVID-1164.png,True,G00183,val,8d5d25ecb31ffb8ab9d76fe0000f459e,996d061a4d5f194f
COVID-1165.png,COVID,COVID/images/COVID-1165.png,COVID/masks/COVID-1165.png,True,G00184,train,942eb5c3e8bd86f0ea61ed0da4308c0d,9979c7961e464978
COVID-1166.png,COVID,COVID/images/COVID-1166.png,COVID/masks/COVID-1166.png,True,G00185,train,2bec6f706645b9f60a24614074d9fc04,d52f7a5909e26899
COVID-1167.png,COVID,COVID/images/COVID-1167.png,COVID/masks/COVID-1167.png,True,G00186,train,d64d204ebc6b0cc22cd71bc293468160,896fc65b4d1139a6
COVID-1168.png,COVID,COVID/images/COVID-1168.png,COVID/masks/COVID-1168.png,True,G00187,train,6d42f2a7e580e776123954148a3075b0,9c6f461e494d384f
COVID-1169.png,COVID,COVID/images/COVID-1169.png,COVID/masks/COVID-1169.png,True,G00188,train,e3e676a4d2a1a5ecdb83a0266c495597,d979d71647525a48
COVID-117.png,COVID,COVID/images/COVID-117.png,COVID/masks/COVID-117.png,True,G00189,train,db8956b718366177cb15895fca3fc001,916be595c4e4e4e4
COVID-1170.png,COVID,COVID/images/COVID-1170.png,COVID/masks/COVID-1170.png,True,G00190,train,00f8d365be16a40eb175cf5bbf45263c,dc6e57182c0b94e7
COVID-1171.png,COVID,COVID/images/COVID-1171.png,COVID/masks/COVID-1171.png,True,G00191,train,7324b5e1d6aa857631a342ad4f7ef9bd,d86c465e1a0f1d5e
COVID-1172.png,COVID,COVID/images/COVID-1172.png,COVID/masks/COVID-1172.png,True,G00192,val,4ddad15f91ae09e7daa69de0e3553cc2,837a5c707b6b61a4
COVID-1173.png,COVID,COVID/images/COVID-1173.png,COVID/masks/COVID-1173.png,True,G00193,train,f71982873ca5b5af3f7522ab237423d8,dd7dc41ac71e8207
COVID-1174.png,COVID,COVID/images/COVID-1174.png,COVID/masks/COVID-1174.png,True,G00194,train,23a124b13e27643b91f2a58a3bf6d5e8,916b56185b7d38a5
COVID-1175.png,COVID,COVID/images/COVID-1175.png,COVID/masks/COVID-1175.png,True,G00195,train,212b19d71a965fa6b902a02a44370662,d17b46c65a196d49
COVID-1176.png,COVID,COVID/images/COVID-1176.png,COVID/masks/COVID-1176.png,True,G00196,train,8f2434f98010bcdbf3b6b1c995f350c3,d02fc71e5c49387a
COVID-1177.png,COVID,COVID/images/COVID-1177.png,COVID/masks/COVID-1177.png,True,G00197,val,21ca720beafe758279ed1e4386c88501,dc1c61c31e5bd2c6
COVID-1178.png,COVID,COVID/images/COVID-1178.png,COVID/masks/COVID-1178.png,True,G00198,test,b573784cab70cdb88f06298422547aaa,dc2d4b0f2c2c3d35
COVID-1179.png,COVID,COVID/images/COVID-1179.png,COVID/masks/COVID-1179.png,True,G00199,train,828ca059fc62fb7a7edf76705105cc32,cd03920b4b5b5b1f
COVID-118.png,COVID,COVID/images/COVID-118.png,COVID/masks/COVID-118.png,True,G00200,test,45c26926977a3e0cc29e9db92cf01bcf,9979c55e1e461a3a
COVID-1180.png,COVID,COVID/images/COVID-1180.png,COVID/masks/COVID-1180.png,True,G00201,test,035d3103df2497b33a462fd38f8e1761,d03dc31fde033c68
COVID-1181.png,COVID,COVID/images/COVID-1181.png,COVID/masks/COVID-1181.png,True,G00202,train,31b3d04d21f8973559fc76655fd7fb19,813b127b791b3a3a
COVID-1182.png,COVID,COVID/images/COVID-1182.png,COVID/masks/COVID-1182.png,True,G00203,train,be968c1586f6900c21414efe4b7403f9,d52a42431b1f4f5b
COVID-1183.png,COVID,COVID/images/COVID-1183.png,COVID/masks/COVID-1183.png,True,G00204,val,99060b968b4116d340dfa91ad743e3a6,d939560ef0293de1
COVID-1184.png,COVID,COVID/images/COVID-1184.png,COVID/masks/COVID-1184.png,True,G00205,train,2fcd2dfe4e1cfc126ae95b65d2203a8d,d5962ad06acbc92b
COVID-1185.png,COVID,COVID/images/COVID-1185.png,COVID/masks/COVID-1185.png,True,G00206,train,2b6dd6cdccc24b69ec49f507d7e2e385,9d48075b1f690d7c
COVID-1186.png,COVID,COVID/images/COVID-1186.png,COVID/masks/COVID-1186.png,True,G00207,train,c2231f20a550f69b1c8c97a9a5064735,bc64d21b474a9a4f
COVID-1187.png,COVID,COVID/images/COVID-1187.png,COVID/masks/COVID-1187.png,True,G00208,test,15c291194063fb476a26930c177f4905,983d465a194f7c4d
COVID-1188.png,COVID,COVID/images/COVID-1188.png,COVID/masks/COVID-1188.png,True,G00209,test,2def6f314819e4cf4a3cf621e3cb8237,d939d6d65a485859
COVID-1189.png,COVID,COVID/images/COVID-1189.png,COVID/masks/COVID-1189.png,True,G00210,train,cc0e82e4687668dd4f6eeb366110bf8d,997d851ec5079a87
COVID-119.png,COVID,COVID/images/COVID-119.png,COVID/masks/COVID-119.png,True,G00211,train,96cadc73ae0184ae2cd0623340e051a5,d07d430b1e0f7b19
COVID-1190.png,COVID,COVID/images/COVID-1190.png,COVID/masks/COVID-1190.png,True,G00212,train,03ed8e4c23a14878b96ae938dac40ac3,887c075a1d6d6c7c
COVID-1191.png,COVID,COVID/images/COVID-1191.png,COVID/masks/COVID-1191.png,True,G00213,train,54e5361fad0b2821a9498530487af584,9979861e1c5b1b59
COVID-1192.png,COVID,COVID/images/COVID-1192.png,COVID/masks/COVID-1192.png,True,G00214,train,f572b0ce92b1b95eee167adca62fc85a,f90dc412155b1f4f
COVID-1193.png,COVID,COVID/images/COVID-1193.png,COVID/masks/COVID-1193.png,True,G00215,train,bcb5f1af87c6cec46ccbfbc0924adda4,d16b1ecc3487649e
COVID-1194.png,COVID,COVID/images/COVID-1194.png,COVID/masks/COVID-1194.png,True,G00216,val,0d88376be2a9e05e072c97ff644506d4,d869e5163ee530e4
COVID-1195.png,COVID,COVID/images/COVID-1195.png,COVID/masks/COVID-1195.png,True,G00217,train,4b8e887e2937b82adfdf60d1581f031c,d82dc7161b471e5a
COVID-1196.png,COVID,COVID/images/COVID-1196.png,COVID/masks/COVID-1196.png,True,G00218,test,ae71ba147282916df866c5b2e0ecbefd,d929063e594e391f
COVID-1197.png,COVID,COVID/images/COVID-1197.png,COVID/masks/COVID-1197.png,True,G00219,train,a033900abfaf33261e45404b95ba518f,dd6dc612475a4b43
COVID-1198.png,COVID,COVID/images/COVID-1198.png,COVID/masks/COVID-1198.png,True,G00220,val,002dda9d17ac4807bf7ee1c79a80eefc,81781e1e5bd9d8f0
COVID-1199.png,COVID,COVID/images/COVID-1199.png,COVID/masks/COVID-1199.png,True,G00221,train,09a26cfdd1406b6010f77e39a41088d0,817b07061f1f5a5d
COVID-12.png,COVID,COVID/images/COVID-12.png,COVID/masks/COVID-12.png,True,G00222,train,7caee4a0e14e60414644eaf2f0a83aac,d43fc30b5c43594d
COVID-120.png,COVID,COVID/images/COVID-120.png,COVID/masks/COVID-120.png,True,G00223,train,c19e12a8c53c79e7e566a02cd89bdfaf,883dc61e1a4f5b3a
COVID-1200.png,COVID,COVID/images/COVID-1200.png,COVID/masks/COVID-1200.png,True,G00177,val,7fa37c2b110aec57a78c48840ec2cce8,8d75d21a671e1b45
COVID-1201.png,COVID,COVID/images/COVID-1201.png,COVID/masks/COVID-1201.png,True,G00224,train,d5eb01ce2c9bae0019e5f2c55e60441c,d43d0b435f0cce5c
COVID-1202.png,COVID,COVID/images/COVID-1202.png,COVID/masks/COVID-1202.png,True,G00225,train,1ecb1916b0284a541750d270abf5995b,9c0f478f4d8c2eaa
COVID-1203.png,COVID,COVID/images/COVID-1203.png,COVID/masks/COVID-1203.png,True,G00226,train,b4da34e10bc6d3697ae581457266503b,887d67928e9f1949
COVID-1204.png,COVID,COVID/images/COVID-1204.png,COVID/masks/COVID-1204.png,True,G00227,val,7d5470dd0a6873db02f0a9938487ff2b,d92f461e490d5d2b
COVID-1205.png,COVID,COVID/images/COVID-1205.png,COVID/masks/COVID-1205.png,True,G00228,train,4f14d819302cc86a29f7d458e2430192,987b47161e197959
COVID-1206.png,COVID,COVID/images/COVID-1206.png,COVID/masks/COVID-1206.png,True,G00229,train,4c5d543bff3f08e83ecc6bcb3df273b3,d57fc21a0b4a1b4b
COVID-1207.png,COVID,COVID/images/COVID-1207.png,COVID/masks/COVID-1207.png,True,G00230,train,cee42cd3774caa60cccb13147ee53283,812be6de584d585d
COVID-1208.png,COVID,COVID/images/COVID-1208.png,COVID/masks/COVID-1208.png,True,G00231,train,a8ed38306c2757f6eb91c97a200ad760,c16f565258107f6d
COVID-1209.png,COVID,COVID/images/COVID-1209.png,COVID/masks/COVID-1209.png,True,G00232,train,3cf82e21e729dfd4811c25cf6816d923,9178175a1d7938ec
COVID-121.png,COVID,COVID/images/COVID-121.png,COVID/masks/COVID-121.png,True,G00233,train,ec0ca0d3e27148352df55df1af458af1,99a8a62a9a875f17
COVID-1210.png,COVID,COVID/images/COVID-1210.png,COVID/masks/COVID-1210.png,True,G00234,val,dc2a148a660f3d3060bd0429162137b3,d42f1b0a2f463df0
COVID-1211.png,COVID,COVID/images/COVID-1211.png,COVID/masks/COVID-1211.png,True,G00235,train,e788197f32c9d60d8d00c85931604b0e,ad24921bc11f4f4f
COVID-1212.png,COVID,COVID/images/COVID-1212.png,COVID/masks/COVID-1212.png,True,G00236,train,5ffe09b891adbd14967a8dfcbc95203e,d4a42bc33b3dc42e
COVID-1213.png,COVID,COVID/images/COVID-1213.png,COVID/masks/COVID-1213.png,True,G00237,val,2cb10e1262b6223e62c8a493e4b1fa39,c86835177b691d1e
COVID-1214.png,COVID,COVID/images/COVID-1214.png,COVID/masks/COVID-1214.png,True,G00238,test,18cef21725e4ddd77719a8553ce5e9f1,d42e424b0b4b2f3f
COVID-1215.png,COVID,COVID/images/COVID-1215.png,COVID/masks/COVID-1215.png,True,G00239,test,06ea0cec8ef464ec54f9c220b87388af,8c3d925a436c7c4f
COVID-1216.png,COVID,COVID/images/COVID-1216.png,COVID/masks/COVID-1216.png,True,G00240,test,fdac6058d993c47700841a4ceaa77112,d07f42194e2f382f
COVID-1217.png,COVID,COVID/images/COVID-1217.png,COVID/masks/COVID-1217.png,True,G00241,train,3a509ecd964f021031ece925f59cea63,d929c417c51c5f5a
COVID-1218.png,COVID,COVID/images/COVID-1218.png,COVID/masks/COVID-1218.png,True,G00242,train,07e5721526b92350c046a5d67500ef15,806987061e1f3fbb
COVID-1219.png,COVID,COVID/images/COVID-1219.png,COVID/masks/COVID-1219.png,True,G00243,train,13921a44cf72c22eeb126fbf536d72ae,992dc517c6665ae2
COVID-122.png,COVID,COVID/images/COVID-122.png,COVID/masks/COVID-122.png,True,G00244,train,12bb9c568a910929db1369856cc1e76b,83789cd2dad0f0ce
COVID-1220.png,COVID,COVID/images/COVID-1220.png,COVID/masks/COVID-1220.png,True,G00245,train,c132fd546e90e558a9f4a54bf5865368,817f165e1a195f1a
COVID-1221.png,COVID,COVID/images/COVID-1221.png,COVID/masks/COVID-1221.png,True,G00246,test,7f1ed0fcfba54ef9cefef518728ee05f,d925861e461f5e4d
COVID-1222.png,COVID,COVID/images/COVID-1222.png,COVID/masks/COVID-1222.png,True,G00247,test,a6b96cd7aa3860849b59078a040657dd,d87fc51ec67e00e0
COVID-1223.png,COVID,COVID/images/COVID-1223.png,COVID/masks/COVID-1223.png,True,G00248,train,d07937beb56441f026c022cb7a417983,817d8616194f177b
COVID-1224.png,COVID,COVID/images/COVID-1224.png,COVID/masks/COVID-1224.png,True,G00249,train,f88b3eb74e92008fa9d5d6f11c1c8436,9c6fc55fc3584a48
COVID-1225.png,COVID,COVID/images/COVID-1225.png,COVID/masks/COVID-1225.png,True,G00250,train,24d25db205e66c82805b94f242021d87,d02b473cb8d3d592
COVID-1226.png,COVID,COVID/images/COVID-1226.png,COVID/masks/COVID-1226.png,True,G00251,train,a09efc21f8400b9351eb89837474d91e,de09cb0f38353f30
COVID-1227.png,COVID,COVID/images/COVID-1227.png,COVID/masks/COVID-1227.png,True,G00252,train,cf1a0c177da49f05090ae0cd5a6f4cc5,d42bc34d2c2e2de6
COVID-1228.png,COVID,COVID/images/COVID-1228.png,COVID/masks/COVID-1228.png,True,G00253,train,a33f27466db74abda8b354f599a6f233,d92f861e04175e5e
COVID-1229.png,COVID,COVID/images/COVID-1229.png,COVID/masks/COVID-1229.png,True,G00254,train,6cce3d8f63103bbaf1491b041e443905,8e29452e339d75d2
COVID-123.png,COVID,COVID/images/COVID-123.png,COVID/masks/COVID-123.png,True,G00255,train,549a0fcfedf09ccdc5be5a50f3611ba8,dd25d20fd20bd668
COVID-1230.png,COVID,COVID/images/COVID-1230.png,COVID/masks/COVID-1230.png,True,G00256,train,e9d6fd0be2ed053be5d58bfe7232fd82,dc21532e2a3535f5
COVID-1231.png,COVID,COVID/images/COVID-1231.png,COVID/masks/COVID-1231.png,True,G00257,val,a5026809301a364475cb0ea602552c60,ad44d21b4f4e0fe1
COVID-1232.png,COVID,COVID/images/COVID-1232.png,COVID/masks/COVID-1232.png,True,G00258,val,ce730df061c0f493434c9a43085f7cda,8c27d30f36d075d2
COVID-1233.png,COVID,COVID/images/COVID-1233.png,COVID/masks/COVID-1233.png,True,G00259,val,478c38c375855cff99e7e13a20b2624c,9869c71e4ec21eda
COVID-1234.png,COVID,COVID/images/COVID-1234.png,COVID/masks/COVID-1234.png,True,G00260,test,d56fb4f2904f9ea199961467507ee19b,d56213293e3a273b
COVID-1235.png,COVID,COVID/images/COVID-1235.png,COVID/masks/COVID-1235.png,True,G00261,train,1d890c47a43c7de3e2dc735ab2c4dc37,9c6d431e491e1f4b
COVID-1236.png,COVID,COVID/images/COVID-1236.png,COVID/masks/COVID-1236.png,True,G00262,train,de7b828d77ef8925d8157f2e8695906f,dd2cc25bd9431e49
COVID-1237.png,COVID,COVID/images/COVID-1237.png,COVID/masks/COVID-1237.png,True,G00263,train,fb805aebd50fde2d14ab0b03876b209b,8f4dc0d2cbdb1991
COVID-1238.png,COVID,COVID/images/COVID-1238.png,COVID/masks/COVID-1238.png,True,G00264,train,d27602a6b99c085da83c1bea2bdeab01,c13dd240bf40ff54
COVID-1239.png,COVID,COVID/images/COVID-1239.png,COVID/masks/COVID-1239.png,True,G00265,train,79a6afbb28c0c6cce7f65d8226af56ad,d92996164f5a1e56
COVID-124.png,COVID,COVID/images/COVID-124.png,COVID/masks/COVID-124.png,True,G00266,train,ee23d421abcb09f3d401546cd056c475,d979d61655535a48
COVID-1240.png,COVID,COVID/images/COVID-1240.png,COVID/masks/COVID-1240.png,True,G00267,train,fb98ce115029efacec1ca7426eb21ff0,c07f434b1a4b3b69
COVID-1241.png,COVID,COVID/images/COVID-1241.png,COVID/masks/COVID-1241.png,True,G00268,val,af132fe545c8588b605bc3090ff0bf78,996d961e441b4f1e
COVID-1242.png,COVID,COVID/images/COVID-1242.png,COVID/masks/COVID-1242.png,True,G00269,train,46c66efefae8cbc1bf5dbd3b1c8e625b,c10ceeda1ec449bd
COVID-1243.png,COVID,COVID/images/COVID-1243.png,COVID/masks/COVID-1243.png,True,G00270,train,f2eee66af45b3ef1714048faf45b51bc,d43ad35d544b1a4b
COVID-1244.png,COVID,COVID/images/COVID-1244.png,COVID/masks/COVID-1244.png,True,G00271,test,1986b7ec4318999a76a5a55249acf432,c947c35b5b415a4e
COVID-1245.png,COVID,COVID/images/COVID-1245.png,COVID/masks/COVID-1245.png,True,G00272,train,0e139524a0253c5ac554df56b6bc9a53,c92e021f1ed87fd0
COVID-1246.png,COVID,COVID/images/COVID-1246.png,COVID/masks/COVID-1246.png,True,G00273,train,db1cbb5e6381e7a8b9ad0226aa698e78,913f861a3d157c55
COVID-1247.png,COVID,COVID/images/COVID-1247.png,COVID/masks/COVID-1247.png,True,G00274,val,b0d51c5e68c7967a19ab10058056c266,d939051696c7163f
COVID-1248.png,COVID,COVID/images/COVID-1248.png,COVID/masks/COVID-1248.png,True,G00275,train,b192201d7e2b66c37aba660b344a29e5,d03d42061f5f0e3f
COVID-1249.png,COVID,COVID/images/COVID-1249.png,COVID/masks/COVID-1249.png,True,G00276,train,f69182973f5b1e2c5b654fec8a942f29,d42f835f4c467c4c
COVID-125.png,COVID,COVID/images/COVID-125.png,COVID/masks/COVID-125.png,True,G00277,train,b7fefa242770eb5094a9eb1a97e980d2,c467434b2cf6f0d1
COVID-1250.png,COVID,COVID/images/COVID-1250.png,COVID/masks/COVID-1250.png,True,G00278,train,02767332eb598ec9bb70f0b8822d5f6b,d929c596c7521f49
COVID-1251.png,COVID,COVID/images/COVID-1251.png,COVID/masks/COVID-1251.png,True,G00279,train,a17ef39e60018601936974ec3482ae75,802d061e2cbdbabf
COVID-1252.png,COVID,COVID/images/COVID-1252.png,COVID/masks/COVID-1252.png,True,G00280,train,b07087349f93957274eebaf72966a7cb,d32b7e854e1f2891
COVID-1253.png,COVID,COVID/images/COVID-1253.png,COVID/masks/COVID-1253.png,True,G00281,train,55b6c7b4e0d076e6b9131846b1ef54ed,9829e686c51bd39e
COVID-1254.png,COVID,COVID/images/COVID-1254.png,COVID/masks/COVID-1254.png,True,G00282,val,d6220f5862e5a942132461904e649f33,d1690f1e5a696978
COVID-1255.png,COVID,COVID/images/COVID-1255.png,COVID/masks/COVID-1255.png,True,G00283,test,84442f7b9331f33e452f6b2e367cb165,937bc5851c01d73e
COVID-1256.png,COVID,COVID/images/COVID-1256.png,COVID/masks/COVID-1256.png,True,G00284,train,610c675c71cbad1155da2d5ad90de68a,d87bc7061e0a313f
COVID-1257.png,COVID,COVID/images/COVID-1257.png,COVID/masks/COVID-1257.png,True,G00285,test,f3f49f043089644761b58d0e537eff4d,912e06596979636f
COVID-1258.png,COVID,COVID/images/COVID-1258.png,COVID/masks/COVID-1258.png,True,G00286,train,bbf17745a20ee4304c3a3222fdb51bb0,c66240482e6e7bf7
COVID-1259.png,COVID,COVID/images/COVID-1259.png,COVID/masks/COVID-1259.png,True,G00287,train,fb3cbc3dd1491b565e79dde7f3a5885b,8c7f42421f1f691d
COVID-126.png,COVID,COVID/images/COVID-126.png,COVID/masks/COVID-126.png,True,G00288,train,5ab3e886c4efe2a6e9d72d7ff1c10dd6,883f831b290d1f3f
COVID-1260.png,COVID,COVID/images/COVID-1260.png,COVID/masks/COVID-1260.png,True,G00289,train,b7438a0d889940b465ca694fa6170243,816806195f7c7a5f
COVID-1261.png,COVID,COVID/images/COVID-1261.png,COVID/masks/COVID-1261.png,True,G00290,train,2de0f37d9bce11b5c64d8dfa01a4b63f,9d2cc31b46471b6d
COVID-1262.png,COVID,COVID/images/COVID-1262.png,COVID/masks/COVID-1262.png,True,G00291,train,b139a2624e4346d8bfaeaaf619a02e0a,907be7a69646ac46
COVID-1263.png,COVID,COVID/images/COVID-1263.png,COVID/masks/COVID-1263.png,True,G00292,train,9ebc4dc2fe072a79b5944ca40fbe24d4,d42a514d4f4e4a5f
COVID-1264.png,COVID,COVID/images/COVID-1264.png,COVID/masks/COVID-1264.png,True,G00293,train,ce1fabbe6c2d2877b8419940ed20ffb6,8c24d25bc51f5b4b
COVID-1265.png,COVID,COVID/images/COVID-1265.png,COVID/masks/COVID-1265.png,True,G00294,val,2a6d05760115b8df46f85d5f6618cccf,d868e2d69e4ae534
COVID-1266.png,COVID,COVID/images/COVID-1266.png,COVID/masks/COVID-1266.png,True,G00295,val,139e43f62de43ac1ce2fafd37543b7cc,917b861a187f193b
COVID-1267.png,COVID,COVID/images/COVID-1267.png,COVID/masks/COVID-1267.png,True,G00296,train,acd8db3217164b704dfce0d3520e6087,d8798587c71f034b
COVID-1268.png,COVID,COVID/images/COVID-1268.png,COVID/masks/COVID-1268.png,True,G00297,test,689938b2b0828501992d7428db92329c,8f2cc452d15f164f
COVID-1269.png,COVID,COVID/images/COVID-1269.png,COVID/masks/COVID-1269.png,True,G00298,test,38fddd1d0a2aef2643afd790dfe7ac2f,d43cc3c3195f4659
COVID-127.png,COVID,COVID/images/COVID-127.png,COVID/masks/COVID-127.png,True,G00299,train,db54ac5dcc1c7b6c749db25d29fb86a7,dd4dc65647524b43
COVID-1270.png,COVID,COVID/images/COVID-1270.png,COVID/masks/COVID-1270.png,True,G00300,val,387dd21c0b6959d09bb7a1715a700d70,8978061a5f4e5f1e
COVID-1271.png,COVID,COVID/images/COVID-1271.png,COVID/masks/COVID-1271.png,True,G00301,train,7f09ff6744a3a651bc9407bc8b8b7d02,d87cf41243475acb
COVID-1272.png,COVID,COVID/images/COVID-1272.png,COVID/masks/COVID-1272.png,True,G00302,test,b7db35337f887b022e023dfde8f6bdf0,ce2d490e0f642e9f
COVID-1273.png,COVID,COVID/images/COVID-1273.png,COVID/masks/COVID-1273.png,True,G00303,train,89e762e338c730e9d34b441a87ee00c9,9c6cc313851dcb4f
COVID-1274.png,COVID,COVID/images/COVID-1274.png,COVID/masks/COVID-1274.png,True,G00304,train,32ee2981ba42ee53c9b38a6455aa33eb,d979a5865e4a1e1a
COVID-1275.png,COVID,COVID/images/COVID-1275.png,COVID/masks/COVID-1275.png,True,G00305,train,df53671fa1febe90cb257a7a8a4f8980,dc2cd30f4e0b1f68
COVID-1276.png,COVID,COVID/images/COVID-1276.png,COVID/masks/COVID-1276.png,True,G00306,train,1fc1eb0dbba953df731c91b1a0b2f9ca,cc6cc012cf4f0f4f
COVID-1277.png,COVID,COVID/images/COVID-1277.png,COVID/masks/COVID-1277.png,True,G00307,train,51bc9eecd1db77503ad6665168bac20f,dd6db4124b991a4b
COVID-1278.png,COVID,COVID/images/COVID-1278.png,COVID/masks/COVID-1278.png,True,G00308,train,54201a76def047959b77f96bb55d6bc1,9968071457c73b1f
COVID-1279.png,COVID,COVID/images/COVID-1279.png,COVID/masks/COVID-1279.png,True,G00309,train,9687ecdcc0e47b274659c857794fb2ee,9958d61f5b1a1e1c
COVID-128.png,COVID,COVID/images/COVID-128.png,COVID/masks/COVID-128.png,True,G00310,train,e7ab9beb8862b4d95047aec009d01e27,d56e024f09296f6e
COVID-1280.png,COVID,COVID/images/COVID-1280.png,COVID/masks/COVID-1280.png,True,G00311,train,4fe18179c3dd2a48aff5793f1adf9a92,fb79c41607121acf
COVID-1281.png,COVID,COVID/images/COVID-1281.png,COVID/masks/COVID-1281.png,True,G00312,train,b758a7a88b41f95d2b01b1a664e46d98,dd2a931f0e832d87
COVID-1282.png,COVID,COVID/images/COVID-1282.png,COVID/masks/COVID-1282.png,True,G00313,train,faaeeac870287ce8d43a78a75822a947,d83995961d505e5e
COVID-1283.png,COVID,COVID/images/COVID-1283.png,COVID/masks/COVID-1283.png,True,G00314,train,0879cdfc7ac949783201bbff2bf8d75c,d809c75acd581b4f
COVID-1284.png,COVID,COVID/images/COVID-1284.png,COVID/masks/COVID-1284.png,True,G00315,val,2566009e1b37586c1cb7d6c5119ecd56,cd0a161a7b79e8e2
COVID-1285.png,COVID,COVID/images/COVID-1285.png,COVID/masks/COVID-1285.png,True,G00316,val,7e4a66a5f465c5e1749e5786b2c2962f,d96d8416c51e5b4e
COVID-1286.png,COVID,COVID/images/COVID-1286.png,COVID/masks/COVID-1286.png,True,G00317,val,dfcb23ace31c521462c68e11a6569a93,8c65c21ecf1d1ec3
COVID-1287.png,COVID,COVID/images/COVID-1287.png,COVID/masks/COVID-1287.png,True,G00318,test,6ec5429c97db735c5fd030e81f24aaf4,947c2245bb171757
COVID-1288.png,COVID,COVID/images/COVID-1288.png,COVID/masks/COVID-1288.png,True,G00319,test,9875fcde732acf2f61e3d687a01945c2,9861e71ee51c1b59
COVID-1289.png,COVID,COVID/images/COVID-1289.png,COVID/masks/COVID-1289.png,True,G00320,train,b87e9a5e2cb5c0c45cd1236c6a04cb2b,f83d971e47461ac1
COVID-129.png,COVID,COVID/images/COVID-129.png,COVID/masks/COVID-129.png,True,G00321,test,d2114dc98e7ca482b41aaa12fbdb1832,817b47161a4d5f1b
COVID-1290.png,COVID,COVID/images/COVID-1290.png,COVID/masks/COVID-1290.png,True,G00322,train,3bc48a54c698108bb5e69ca8074ddaba,d42f431b4f6e0c4e
COVID-1291.png,COVID,COVID/images/COVID-1291.png,COVID/masks/COVID-1291.png,True,G00323,train,087d549f0dab465558706d0a9c91a169,dc6c821a871e1f5e
COVID-1292.png,COVID,COVID/images/COVID-1292.png,COVID/masks/COVID-1292.png,True,G00324,val,b9431650c5ac614b6c8c323cab60443a,bd65d21ac51a4b4b
COVID-1293.png,COVID,COVID/images/COVID-1293.png,COVID/masks/COVID-1293.png,True,G00325,train,171c74db6ab3c0f4acda40c485fa30d6,9378041e185f7b79
COVID-1294.png,COVID,COVID/images/COVID-1294.png,COVID/masks/COVID-1294.png,True,G00326,train,073b8a0b27d2c6ed191ccada0828c0ab,8c7d67c2c6971a83
COVID-1295.png,COVID,COVID/images/COVID-1295.png,COVID/masks/COVID-1295.png,True,G00327,train,8220d431ba32411c3e2d106737c84554,d42ec3119bc3673a
COVID-1296.png,COVID,COVID/images/COVID-1296.png,COVID/masks/COVID-1296.png,True,G00328,train,aa23b159e796309079642e8fc7d578d0,d87dc516851e1e4b
COVID-1297.png,COVID,COVID/images/COVID-1297.png,COVID/masks/COVID-1297.png,True,G00329,val,def2af888c7e129f47bcb06412bd9357,9879e79696121a5b
COVID-1298.png,COVID,COVID/images/COVID-1298.png,COVID/masks/COVID-1298.png,True,G00330,train,2a4aa520cff25c9ab72a26f162c3e24a,916b861a8657395f
COVID-1299.png,COVID,COVID/images/COVID-1299.png,COVID/masks/COVID-1299.png,True,G00331,train,feeece6fe5f6406e47f31f4aaa3313e3,d82dc61ac51f194f
COVID-13.png,COVID,COVID/images/COVID-13.png,COVID/masks/COVID-13.png,True,G00332,test,982261eb78353b6e38bc7d048902891d,9d6e860b0b455d7c
COVID-130.png,COVID,COVID/images/COVID-130.png,COVID/masks/COVID-130.png,True,G00333,train,e3bfc96f1afee24dfeda164c523cb43b,d925861e071f5e4d
COVID-1300.png,COVID,COVID/images/COVID-1300.png,COVID/masks/COVID-1300.png,True,G00334,train,9e14e00ebf3399ac09422e592fe5aaea,917bc5d4be04b616
COVID-1301.png,COVID,COVID/images/COVID-1301.png,COVID/masks/COVID-1301.png,True,G00335,train,0440b965a6ee668ff11680e6126eb57a,d12946d7182f5759
COVID-1302.png,COVID,COVID/images/COVID-1302.png,COVID/masks/COVID-1302.png,True,G00336,train,a2a463766535ce6c6248ac39c46b21dd,ec7d52534b4b4d0c
COVID-1303.png,COVID,COVID/images/COVID-1303.png,COVID/masks/COVID-1303.png,True,G00337,train,3850def98e2538627e17466d8e6c8693,cc66e21bc9d8d9c4
COVID-1304.png,COVID,COVID/images/COVID-1304.png,COVID/masks/COVID-1304.png,True,G00338,train,c4256cc19b1e4481900970e30f40dee5,d42a430d4c6e5aef
COVID-1305.png,COVID,COVID/images/COVID-1305.png,COVID/masks/COVID-1305.png,True,G00339,test,e7f6750b3db7549829552fc9a08c553e,d939c4165f491e5a
COVID-1306.png,COVID,COVID/images/COVID-1306.png,COVID/masks/COVID-1306.png,True,G00340,train,88f72930cb6dc7c35b6a0250624a403e,9d51c61e451e594f
COVID-1307.png,COVID,COVID/images/COVID-1307.png,COVID/masks/COVID-1307.png,True,G00341,train,0515f8fcd1a555ba0d5a9809749dda5b,d43c43c31b4f4f49
COVID-1308.png,COVID,COVID/images/COVID-1308.png,COVID/masks/COVID-1308.png,True,G00342,train,1ad9e06ab76b7758395e357926d011ae,d96ad7921f434a1c
COVID-1309.png,COVID,COVID/images/COVID-1309.png,COVID/masks/COVID-1309.png,True,G00343,val,156a6c95a919c43f673b58a4a3c21ded,8f6fe01a1f650c0f
COVID-131.png,COVID,COVID/images/COVID-131.png,COVID/masks/COVID-131.png,True,G00344,test,25d007bd8c624913838c810d04bf76d2,817f74494b4be48e
COVID-1310.png,COVID,COVID/images/COVID-1310.png,COVID/masks/COVID-1310.png,True,G00345,train,e5b836ec28d9a75b0089688a35008e0b,dc2cc30b0fcb0d4f
COVID-1311.png,COVID,COVID/images/COVID-1311.png,COVID/masks/COVID-1311.png,True,G00346,train,2445c4a468fd9ef3f60198161b783957,dd2ec2131f451d0f
COVID-1312.png,COVID,COVID/images/COVID-1312.png,COVID/masks/COVID-1312.png,True,G00347,val,38f2991b00893332c92836933c70c45a,d839c74e94505b5e
COVID-1313.png,COVID,COVID/images/COVID-1313.png,COVID/masks/COVID-1313.png,True,G00348,test,d1ab5eafdd959d27ae9f494274476fc4,8139061f5b5e6b39
COVID-1314.png,COVID,COVID/images/COVID-1314.png,COVID/masks/COVID-1314.png,True,G00349,train,7d11eaac49db88ae981fc8562a6d414f,cd2c1253434b1f6f
COVID-1315.png,COVID,COVID/images/COVID-1315.png,COVID/masks/COVID-1315.png,True,G00350,test,b45684ad2f2d9f8392b9132f76f10fe4,f82d4752131f4d55
COVID-1316.png,COVID,COVID/images/COVID-1316.png,COVID/masks/COVID-1316.png,True,G00351,train,387e151f60b0d6d575d11853fa71e02f,d124577b1e0b382f
COVID-1317.png,COVID,COVID/images/COVID-1317.png,COVID/masks/COVID-1317.png,True,G00352,train,24ce57499e38a635869a9835be80b7c6,986de492e11ec31f
COVID-1318.png,COVID,COVID/images/COVID-1318.png,COVID/masks/COVID-1318.png,True,G00353,train,8a0e3b24cb7ce058fb13445f9b6f509a,9d79c616c31f410f
COVID-1319.png,COVID,COVID/images/COVID-1319.png,COVID/masks/COVID-1319.png,True,G00354,train,1a7fc174cff1a56affed6957d70a6e19,913ac6197c6d395c
COVID-132.png,COVID,COVID/images/COVID-132.png,COVID/masks/COVID-132.png,True,G00355,train,ad43c70db69f846195a92768e6bd40a7,8835d20fd319577c
COVID-1320.png,COVID,COVID/images/COVID-1320.png,COVID/masks/COVID-1320.png,True,G00356,train,3174c8738137940c70162e3185007ba3,9c6f821bc71e194d
COVID-1321.png,COVID,COVID/images/COVID-1321.png,COVID/masks/COVID-1321.png,True,G00357,train,17accd76d797ac32dff5cd651484a6e7,ca09a5f497eb941a
COVID-1322.png,COVID,COVID/images/COVID-1322.png,COVID/masks/COVID-1322.png,True,G00358,val,fd086e73418a14652a1c45bd221a47a7,993bc71a194b3b58
COVID-1323.png,COVID,COVID/images/COVID-1323.png,COVID/masks/COVID-1323.png,True,G00359,val,12f4e27d576a3fe12aa7a5e5ac991d12,9c25c31ec35b1e69
COVID-1324.png,COVID,COVID/images/COVID-1324.png,COVID/masks/COVID-1324.png,True,G00360,test,d74ba546c6e00e64e2ba9fcd249e7714,bc6cc7138ba5388d
COVID-1325.png,COVID,COVID/images/COVID-1325.png,COVID/masks/COVID-1325.png,True,G00361,train,dbbe342b252f0f445028d7fbeb2fe46e,916ec7194c4f194f
COVID-1326.png,COVID,COVID/images/COVID-1326.png,COVID/masks/COVID-1326.png,True,G00362,train,d82396000257a736a8e33d8af6e1a246,cc2dc252db531a5b
COVID-1327.png,COVID,COVID/images/COVID-1327.png,COVID/masks/COVID-1327.png,True,G00363,train,b61898fea6f7be8734895ea8a23f94df,f861c71ec11ddc1c
COVID-1328.png,COVID,COVID/images/COVID-1328.png,COVID/masks/COVID-1328.png,True,G00364,train,16a446731c28781a891632ca5c850844,cc2fd10ccb4d5e68
COVID-1329.png,COVID,COVID/images/COVID-1329.png,COVID/masks/COVID-1329.png,True,G00365,train,361e73f76187345abfc333ebd020e8da,9979851f14561e3d
COVID-133.png,COVID,COVID/images/COVID-133.png,COVID/masks/COVID-133.png,True,G00366,train,4cfa4925b51fdb985e12822d6f237627,d42f831b5d467c4c
COVID-1330.png,COVID,COVID/images/COVID-1330.png,COVID/masks/COVID-1330.png,True,G00367,train,9bb27ba0ec14b06f0592fd0bea4ac5a6,dc1ee2e38b0dc50d
COVID-1331.png,COVID,COVID/images/COVID-1331.png,COVID/masks/COVID-1331.png,True,G00368,train,27e82fbf215997fe9b11cdabeb417beb,c42dd365168bc91f
COVID-1332.png,COVID,COVID/images/COVID-1332.png,COVID/masks/COVID-1332.png,True,G00369,train,b3f311c7351436f08f5703c0f9df27e1,983dc55657c3461e
COVID-1333.png,COVID,COVID/images/COVID-1333.png,COVID/masks/COVID-1333.png,True,G00370,train,92d101dc1f5d839430eb203566780f1d,cd3cf2139b35282d
COVID-1334.png,COVID,COVID/images/COVID-1334.png,COVID/masks/COVID-1334.png,True,G00371,train,a37516070d8ff3c97f395a5fc5ca4c9e,d52a23411b5d7d3c
COVID-1335.png,COVID,COVID/images/COVID-1335.png,COVID/masks/COVID-1335.png,True,G00142,train,25cb2746588e224eac8495cc9e0dcc65,987d8716174f1707
COVID-1336.png,COVID,COVID/images/COVID-1336.png,COVID/masks/COVID-1336.png,True,G00372,train,3c5e8f1b88d06f7732985971a0204862,cd7cb04b4d535711
COVID-1337.png,COVID,COVID/images/COVID-1337.png,COVID/masks/COVID-1337.png,True,G00373,train,8f6413447454e2159f8b3668539ba2cf,8e4a434f6cb8f4b4
COVID-1338.png,COVID,COVID/images/COVID-1338.png,COVID/masks/COVID-1338.png,True,G00374,train,2cc9fcc61cc162649565683741a5be90,c43cd34b1e6c1b3c
COVID-1339.png,COVID,COVID/images/COVID-1339.png,COVID/masks/COVID-1339.png,True,G00375,val,965d776c1e3ed37162d065738c94c559,dc6ca2969f8d0d0d
COVID-134.png,COVID,COVID/images/COVID-134.png,COVID/masks/COVID-134.png,True,G00376,val,53a420f54f56561904864346834ae191,917b561819a7ace5
COVID-1340.png,COVID,COVID/images/COVID-1340.png,COVID/masks/COVID-1340.png,True,G00377,val,e91233cf8f8f33075a635d8140066a84,907f8659693ceae0
COVID-1341.png,COVID,COVID/images/COVID-1341.png,COVID/masks/COVID-1341.png,True,G00378,test,bdc027a341d6418539885208fb5b6fef,d42e830f4c6eeae1
COVID-1342.png,COVID,COVID/images/COVID-1342.png,COVID/masks/COVID-1342.png,True,G00379,test,badde6337e1890afe67f0dedc9748250,bd61d65ec15ad148
COVID-1343.png,COVID,COVID/images/COVID-1343.png,COVID/masks/COVID-1343.png,True,G00380,train,fa9202d745c33c79415253d7b7065732,917b841e1a1f3f1c
COVID-1344.png,COVID,COVID/images/COVID-1344.png,COVID/masks/COVID-1344.png,True,G00381,train,701581040437176e26704ecbc7f5cddc,987b45471e524b3b
COVID-1345.png,COVID,COVID/images/COVID-1345.png,COVID/masks/COVID-1345.png,True,G00382,train,2800e203a87131ae9182a6f3716feafc,d86da41e87e21b2d
COVID-1346.png,COVID,COVID/images/COVID-1346.png,COVID/masks/COVID-1346.png,True,G00383,test,eec17629d03be199720f0717d4e9b31e,c42e871d155d3c7c
COVID-1347.png,COVID,COVID/images/COVID-1347.png,COVID/masks/COVID-1347.png,True,G00384,train,ed50dc1cba0ff5ea97d61ea8603c34bb,d42e035f0f0a2f4f
COVID-1348.png,COVID,COVID/images/COVID-1348.png,COVID/masks/COVID-1348.png,True,G00385,train,454503161dd7106e4c16ff339af0ac76,9d6ce092eb0d740f
COVID-1349.png,COVID,COVID/images/COVID-1349.png,COVID/masks/COVID-1349.png,True,G00386,train,0dbccf79905c05211aeb71c8a1ad49ae,d43dd34b4b031b1b
COVID-135.png,COVID,COVID/images/COVID-135.png,COVID/masks/COVID-135.png,True,G00387,train,b65f4d64ca62ca965c08813d5cf0cfde,9c2d421fab683c6e
COVID-1350.png,COVID,COVID/images/COVID-1350.png,COVID/masks/COVID-1350.png,True,G00388,train,157eff0f0c31956b3638ff2f2abfe727,d829b59687965a1e
COVID-1351.png,COVID,COVID/images/COVID-1351.png,COVID/masks/COVID-1351.png,True,G00389,train,c39baa266c17db3766934d5cd3f10f98,d82de492c7d093e5
COVID-1352.png,COVID,COVID/images/COVID-1352.png,COVID/masks/COVID-1352.png,True,G00390,test,d901158e301dcb5ed2dbfdf29cfde5c5,8578065f1c4d4b5f
COVID-1353.png,COVID,COVID/images/COVID-1353.png,COVID/masks/COVID-1353.png,True,G00391,train,acf71f15330265200800516cdd4ced93,d07b46c65b197d21
COVID-1354.png,COVID,COVID/images/COVID-1354.png,COVID/masks/COVID-1354.png,True,G00392,train,7641c74d1a5dc2308641ebb53bb07b7a,ec3dc45397d10e0e
COVID-1355.png,COVID,COVID/images/COVID-1355.png,COVID/masks/COVID-1355.png,True,G00393,val,dbc646428051db76a524de4fd788d629,c43d4b2e3536b3b0
COVID-1356.png,COVID,COVID/images/COVID-1356.png,COVID/masks/COVID-1356.png,True,G00394,train,6e24959ff0eb6de2a71b49619ef6445b,c83975021736567f
COVID-1357.png,COVID,COVID/images/COVID-1357.png,COVID/masks/COVID-1357.png,True,G00395,train,e2ac5af8a344f6b6f7a58996a803a052,cd36fa03db83922c
COVID-1358.png,COVID,COVID/images/COVID-1358.png,COVID/masks/COVID-1358.png,True,G00396,val,f7bb9f5b779d0019b079db022d5ce70a,9178861a15577a3f
COVID-1359.png,COVID,COVID/images/COVID-1359.png,COVID/masks/COVID-1359.png,True,G00397,test,25332c2a711fdadda510ab14ef05f1e0,d83166c24eda1fd8
COVID-136.png,COVID,COVID/images/COVID-136.png,COVID/masks/COVID-136.png,True,G00398,train,3fb12910fd8456f030da31c64c4448f9,d5ae861a1d1e1c5d
COVID-1360.png,COVID,COVID/images/COVID-1360.png,COVID/masks/COVID-1360.png,True,G00399,train,b019c2b026a9bbbc2d9990ccb63ed5c6,996dc71ec1175847
COVID-1361.png,COVID,COVID/images/COVID-1361.png,COVID/masks/COVID-1361.png,True,G00400,val,4af3f789e05d94c301dbd3a66759f8ac,986dc61b471e495e
COVID-1362.png,COVID,COVID/images/COVID-1362.png,COVID/masks/COVID-1362.png,True,G00401,test,7fb8735ccd6c19f6d97f5d70a4abc7ac,d93de7474e0a1839
COVID-1363.png,COVID,COVID/images/COVID-1363.png,COVID/masks/COVID-1363.png,True,G00402,train,0e98ccc552cdb44053f472f185f0e876,d939c5be16cb1e10
COVID-1364.png,COVID,COVID/images/COVID-1364.png,COVID/masks/COVID-1364.png,True,G00403,val,c96372f5a8e7eee16f95dd827b3c2d8b,856e065e5f786968
COVID-1365.png,COVID,COVID/images/COVID-1365.png,COVID/masks/COVID-1365.png,True,G00404,train,39437954cd7b1122616996c387248194,d93b458e56451e1e
COVID-1366.png,COVID,COVID/images/COVID-1366.png,COVID/masks/COVID-1366.png,True,G00405,test,013cfd6e0bfe0b764026d81d47ebc482,d42f074a6e6e686a
COVID-1367.png,COVID,COVID/images/COVID-1367.png,COVID/masks/COVID-1367.png,True,G00145,test,197813fc0a56530e7d5d383cab591b9a,c47e27c33bb468b0
COVID-1368.png,COVID,COVID/images/COVID-1368.png,COVID/masks/COVID-1368.png,True,G00406,train,90b3b2a2950bf22378cf43995cb8770c,cf74c087ab91b919
COVID-1369.png,COVID,COVID/images/COVID-1369.png,COVID/masks/COVID-1369.png,True,G00407,train,f3f27834413e8d7d6d77e383717657d9,d52c065e5d686979
COVID-137.png,COVID,COVID/images/COVID-137.png,COVID/masks/COVID-137.png,True,G00408,train,5cc65acef3c47f8c2999b9fa98c9369b,817b5612585b3b3b
COVID-1370.png,COVID,COVID/images/COVID-1370.png,COVID/masks/COVID-1370.png,True,G00149,train,721fa939269b3faf84225e8202873c49,847f239279e17692
COVID-1371.png,COVID,COVID/images/COVID-1371.png,COVID/masks/COVID-1371.png,True,G00409,train,eb96d0ffd7cab4b6dbceb90f396726a5,f83d2584c207efa6
COVID-1372.png,COVID,COVID/images/COVID-1372.png,COVID/masks/COVID-1372.png,True,G00410,train,fe956ced92557f17c9cef555d2540d62,816886d78a9df939
COVID-1373.png,COVID,COVID/images/COVID-1373.png,COVID/masks/COVID-1373.png,True,G00411,train,df2f227992c1f7bc66f4b8216a28e240,cc4b530f4d2d8fa8
COVID-1374.png,COVID,COVID/images/COVID-1374.png,COVID/masks/COVID-1374.png,True,G00412,train,133e1f20fe2bf432deed3e7428f86b48,dd7de2a23a29e02c
COVID-1375.png,COVID,COVID/images/COVID-1375.png,COVID/masks/COVID-1375.png,True,G00413,train,d2c70e8a895eb19ee8c82768e57cef5c,d42f434a2c6c74f3
COVID-1376.png,COVID,COVID/images/COVID-1376.png,COVID/masks/COVID-1376.png,True,G00414,train,83dca9402fa24b52682cbe571f45d626,d86d561e486d3c3c
COVID-1377.png,COVID,COVID/images/COVID-1377.png,COVID/masks/COVID-1377.png,True,G00415,train,9f0687867ac3e1c072abae3be9ab8078,dc04e3838badad8d
COVID-1378.png,COVID,COVID/images/COVID-1378.png,COVID/masks/COVID-1378.png,True,G00416,train,f48403e4eafb77354e65e137766ef370,856c065e5d6c697c
COVID-1379.png,COVID,COVID/images/COVID-1379.png,COVID/masks/COVID-1379.png,True,G00417,val,b18a8360d9c909369bb1e13aececbae8,d81d43c71e1ad30f
COVID-138.png,COVID,COVID/images/COVID-138.png,COVID/masks/COVID-138.png,True,G00418,val,fc9dc3a10d292cbc7b5ded974c177c88,dc7ee1031f51383c
COVID-1380.png,COVID,COVID/images/COVID-1380.png,COVID/masks/COVID-1380.png,True,G00419,test,e12b35d85f04913e4bbdb2e9d9c3ae46,f839e5e50b8fe120
COVID-1381.png,COVID,COVID/images/COVID-1381.png,COVID/masks/COVID-1381.png,True,G00420,test,0765e166d5287d327221f0da667a939f,883d06d70a5e5c5f
COVID-1382.png,COVID,COVID/images/COVID-1382.png,COVID/masks/COVID-1382.png,True,G00421,train,29a98592b27d908aa6a1789b8bae2516,d929c7d68ec39311
COVID-1383.png,COVID,COVID/images/COVID-1383.png,COVID/masks/COVID-1383.png,True,G00422,train,e3898f052888d6e9b3c1996eb423c953,dc2e434b4c4c727e
COVID-1384.png,COVID,COVID/images/COVID-1384.png,COVID/masks/COVID-1384.png,True,G00423,train,b12fa2c231546a493d662abd279bdc53,d979e516d5c612c2
COVID-1385.png,COVID,COVID/images/COVID-1385.png,COVID/masks/COVID-1385.png,True,G00424,train,f69853381b67a2214f85dfdbce0f5b84,887c065e5c71f3f0
COVID-1386.png,COVID,COVID/images/COVID-1386.png,COVID/masks/COVID-1386.png,True,G00268,val,303c648eb5e169f1a48b32db75d36b22,996d961e441b4f1e
COVID-1387.png,COVID,COVID/images/COVID-1387.png,COVID/masks/COVID-1387.png,True,G00425,train,d2284e41813603a9fdc3673a73d68037,cc2ce3130f4d5d0f
COVID-1388.png,COVID,COVID/images/COVID-1388.png,COVID/masks/COVID-1388.png,True,G00426,train,b6d3782c19129e121e1bbd4decce006c,dd3913430e1c5b5b
COVID-1389.png,COVID,COVID/images/COVID-1389.png,COVID/masks/COVID-1389.png,True,G00427,train,99313e3e919653f5bfd9226ebaeade09,d52d830f1f2c3e0d
COVID-139.png,COVID,COVID/images/COVID-139.png,COVID/masks/COVID-139.png,True,G00408,train,5cc65acef3c47f8c2999b9fa98c9369b,817b5612585b3b3b
COVID-1390.png,COVID,COVID/images/COVID-1390.png,COVID/masks/COVID-1390.png,True,G00428,train,aa03b082c2a1364bbd30edf43557103f,9969c5961e1e131f
COVID-1391.png,COVID,COVID/images/COVID-1391.png,COVID/masks/COVID-1391.png,True,G00429,train,f37244d992054a6f6758c7e37fc2b30e,c4af9219514d5d4f
COVID-1392.png,COVID,COVID/images/COVID-1392.png,COVID/masks/COVID-1392.png,True,G00430,train,1f8244008a1067df9ab1cef485af2d58,8c4f834b4d7d6968
COVID-1393.png,COVID,COVID/images/COVID-1393.png,COVID/masks/COVID-1393.png,True,G00431,train,b39c8c7fa19ad95cee62b03d4deadc20,d979961eca191e16
COVID-1394.png,COVID,COVID/images/COVID-1394.png,COVID/masks/COVID-1394.png,True,G00432,train,644adfa177cc6df8b7f869e9afe9a254,d4bdc21b1a501f5b
COVID-1395.png,COVID,COVID/images/COVID-1395.png,COVID/masks/COVID-1395.png,True,G00433,train,e7b660d7f35d3c9c9f164d47ea68e58c,cc26434b4d4f4f6c
COVID-1396.png,COVID,COVID/images/COVID-1396.png,COVID/masks/COVID-1396.png,True,G00434,val,26f45bd88f12be220b6c89dd50fd5d2f,9c0ce5b68e83951f
COVID-1397.png,COVID,COVID/images/COVID-1397.png,COVID/masks/COVID-1397.png,True,G00435,val,ef5cb4543a5a7aa95e39d59f755c6734,817f06161b536d5b
COVID-1398.png,COVID,COVID/images/COVID-1398.png,COVID/masks/COVID-1398.png,True,G00436,val,6879bb6a2d7b0bc130b897fb4ad3de19,bd2cc21ac7c31bc5
COVID-1399.png,COVID,COVID/images/COVID-1399.png,COVID/masks/COVID-1399.png,True,G00437,train,5df5bb16e592d9c9b2f511c15d44bf10,9818b6c68adbcd59
COVID-14.png,COVID,COVID/images/COVID-14.png,COVID/masks/COVID-14.png,True,G00438,test,7c12039720e57338dc54ba691c2efae2,d87bc5d51a8ed814
COVID-140.png,COVID,COVID/images/COVID-140.png,COVID/masks/COVID-140.png,True,G00000,test,7b0d682211f32513201bed8cede77d11,d17b561a59191b1e
COVID-1400.png,COVID,COVID/images/COVID-1400.png,COVID/masks/COVID-1400.png,True,G00439,test,1c7fe5b6e646057115575e24e02dbf2f,c97f07161a594d59
COVID-1401.png,COVID,COVID/images/COVID-1401.png,COVID/masks/COVID-1401.png,True,G00440,train,616f35953b7e8e5b89de7751d6c00559,9c6da01ae33ccb87
COVID-1402.png,COVID,COVID/images/COVID-1402.png,COVID/masks/COVID-1402.png,True,G00441,test,7d9d991fa33a64ee7fd3e81a21894cfd,9e789547d6568316
COVID-1403.png,COVID,COVID/images/COVID-1403.png,COVID/masks/COVID-1403.png,True,G00442,train,8cdd96266011be852d9ccd013fd52682,c43e9319596c6cba
COVID-1404.png,COVID,COVID/images/COVID-1404.png,COVID/masks/COVID-1404.png,True,G00443,train,2d9da8e7d90a4c26e68b694b3156181f,9c7dc0d207c71e2d
COVID-1405.png,COVID,COVID/images/COVID-1405.png,COVID/masks/COVID-1405.png,True,G00444,train,0360307239984739d0712c2f5dc815cb,cc64d29be3831cf4
COVID-1406.png,COVID,COVID/images/COVID-1406.png,COVID/masks/COVID-1406.png,True,G00445,train,37e87c591d99d307c3f22fabba64a63f,d42efa51085173fa
COVID-1407.png,COVID,COVID/images/COVID-1407.png,COVID/masks/COVID-1407.png,True,G00446,train,fdaa59fa3c8c56b2a1bb598654cc75dd,d52e42431b1f0f5b
COVID-1408.png,COVID,COVID/images/COVID-1408.png,COVID/masks/COVID-1408.png,True,G00447,train,c6cae37b4c6132ecd7a7d0b7ce28c7f6,d929d7865e421e1e
COVID-1409.png,COVID,COVID/images/COVID-1409.png,COVID/masks/COVID-1409.png,True,G00448,test,c597707457ddd4df31e62703ad8efcc2,907f851ca81f3b2b
COVID-141.png,COVID,COVID/images/COVID-141.png,COVID/masks/COVID-141.png,True,G00449,test,53f474b92ade89065f0ac87df9a7c3ea,817d87963a292d3d
COVID-1410.png,COVID,COVID/images/COVID-1410.png,COVID/masks/COVID-1410.png,True,G00450,train,99d69a0d9521d1df8cccf23c953554ec,9c6cc30b4d5c5e4d
COVID-1411.png,COVID,COVID/images/COVID-1411.png,COVID/masks/COVID-1411.png,True,G00451,train,c8cc986cf00a09bcd3aa91346fa4b531,b82ca692db6db06c
COVID-1412.png,COVID,COVID/images/COVID-1412.png,COVID/masks/COVID-1412.png,True,G00452,train,bf6ebc597fc530b98bec8fadff1dfd21,8d0d47061e4d4f5f
COVID-1413.png,COVID,COVID/images/COVID-1413.png,COVID/masks/COVID-1413.png,True,G00159,test,43f7d0d1385c2c3f40de84d0591331ef,bc3d434b1b451e1e
COVID-1414.png,COVID,COVID/images/COVID-1414.png,COVID/masks/COVID-1414.png,True,G00453,train,339c6d9ff42a89387a5c33839f6b5643,dd7fa4765d000955
COVID-1415.png,COVID,COVID/images/COVID-1415.png,COVID/masks/COVID-1415.png,True,G00454,train,a43087e1800f6e57c0f66f9974f12297,997f161818cb38be
COVID-1416.png,COVID,COVID/images/COVID-1416.png,COVID/masks/COVID-1416.png,True,G00139,train,497e1992b44f02620aa7498dad185f60,dd28d3c35f55420b
COVID-1417.png,COVID,COVID/images/COVID-1417.png,COVID/masks/COVID-1417.png,True,G00455,train,5410363b5670297f86b3c191fbc2fc38,f92cc61aaf703c4c
COVID-1418.png,COVID,COVID/images/COVID-1418.png,COVID/masks/COVID-1418.png,True,G00456,train,6d446cb178758cbd92a587358d74bae8,997b86141c477b1e
COVID-1419.png,COVID,COVID/images/COVID-1419.png,COVID/masks/COVID-1419.png,True,G00162,train,7258448750c3f1d998001f9dcf99d435,9d2ce096c37a2f68
COVID-142.png,COVID,COVID/images/COVID-142.png,COVID/masks/COVID-142.png,True,G00457,val,2eb358cb4b77a481d3a8fbf19666a230,893f065a696da4e6
COVID-1420.png,COVID,COVID/images/COVID-1420.png,COVID/masks/COVID-1420.png,True,G00458,val,e7d8ad46bf2e177daa1463db29825d75,f878f187164a585d
COVID-1421.png,COVID,COVID/images/COVID-1421.png,COVID/masks/COVID-1421.png,True,G00459,val,31e6b5d02d0825bbc8b8ed5574e894c8,8d6f435b45594d43
COVID-1422.png,COVID,COVID/images/COVID-1422.png,COVID/masks/COVID-1422.png,True,G00152,train,80fb6cbcb831bab45d18ccd5e4dd0a6b,dc6c535f0d1c0f2a
COVID-1423.png,COVID,COVID/images/COVID-1423.png,COVID/masks/COVID-1423.png,True,G00460,test,016b690ee11a412063a7b0c9ebe85eb9,b869e71e8e69184d
COVID-1424.png,COVID,COVID/images/COVID-1424.png,COVID/masks/COVID-1424.png,True,G00461,train,63c51318a88f9a471c9467d3d0fb1516,cd4f430d412de7a9
COVID-1425.png,COVID,COVID/images/COVID-1425.png,COVID/masks/COVID-1425.png,True,G00462,train,51b7f1465d880d18494efbdbc01c09c5,d92dd65396c64714
COVID-1426.png,COVID,COVID/images/COVID-1426.png,COVID/masks/COVID-1426.png,True,G00463,train,1417d15025e7f5d356da371eb76b90db,982dc7061f4d5e1e
COVID-1427.png,COVID,COVID/images/COVID-1427.png,COVID/masks/COVID-1427.png,True,G00464,train,d6045a8d9dd9fa2e7d248b041ffdee2e,d912561e1a393d2f
COVID-1428.png,COVID,COVID/images/COVID-1428.png,COVID/masks/COVID-1428.png,True,G00465,train,1a71168021b3d073e7f00ade5d77e121,d93f561b444b194e
COVID-1429.png,COVID,COVID/images/COVID-1429.png,COVID/masks/COVID-1429.png,True,G00466,train,8b1d8307a2d3628db25eaec2798e47d5,bc0dc31286d71d1f
COVID-143.png,COVID,COVID/images/COVID-143.png,COVID/masks/COVID-143.png,True,G00467,train,e8f0c9f566a20ae24a2d982e8a571478,c92d551217564b5f
COVID-1430.png,COVID,COVID/images/COVID-1430.png,COVID/masks/COVID-1430.png,True,G00468,train,f19dca25b9bc6e918fa08c3f793ac888,d52ac35b560a5c5b
COVID-1431.png,COVID,COVID/images/COVID-1431.png,COVID/masks/COVID-1431.png,True,G00469,train,4be7a6fe5214a4ae5aa5cf4bd5bb60c1,ed3d1796188907ba
COVID-1432.png,COVID,COVID/images/COVID-1432.png,COVID/masks/COVID-1432.png,True,G00470,train,ac77b48c900d453a68952f35a5595897,d821d40ee618fbf8
COVID-1433.png,COVID,COVID/images/COVID-1433.png,COVID/masks/COVID-1433.png,True,G00471,val,67fd1b00a2645ec015053330ea69b6ef,d916521a1a393d3f
COVID-1434.png,COVID,COVID/images/COVID-1434.png,COVID/masks/COVID-1434.png,True,G00472,val,43fd1496191d9ff8eef4aba04733b48d,905b4fda3be064b1
COVID-1435.png,COVID,COVID/images/COVID-1435.png,COVID/masks/COVID-1435.png,True,G00473,train,e9588e768e91cad5e4e6a34ec520f8fa,d809c51fcbf29ac2
COVID-1436.png,COVID,COVID/images/COVID-1436.png,COVID/masks/COVID-1436.png,True,G00474,test,66fe1eb9938459b918e28f7559cf4d83,db0981171f431f6b
COVID-1437.png,COVID,COVID/images/COVID-1437.png,COVID/masks/COVID-1437.png,True,G00271,test,16b8da1e61bce294aaa7dd8f98f9f8eb,c947c35b5b415a4e
COVID-1438.png,COVID,COVID/images/COVID-1438.png,COVID/masks/COVID-1438.png,True,G00475,test,06c2b651b241438f88930daa6b9d9a35,d869c716ee051e4b
COVID-1439.png,COVID,COVID/images/COVID-1439.png,COVID/masks/COVID-1439.png,True,G00476,test,91fcad414579025cc7cb1c5686c5ba58,d93987161e0d1f1e
COVID-144.png,COVID,COVID/images/COVID-144.png,COVID/masks/COVID-144.png,True,G00477,train,fd01420b756b638b56fdf710951c9c66,d82da55a1a575a1b
COVID-1440.png,COVID,COVID/images/COVID-1440.png,COVID/masks/COVID-1440.png,True,G00478,train,141368bfed63ce7eeb7a417bf215dffd,d939c7961e4a3958
COVID-1441.png,COVID,COVID/images/COVID-1441.png,COVID/masks/COVID-1441.png,True,G00479,train,7c3337cb2e92f55e619f8d0e80761429,da39c5da1e9cd052
COVID-1442.png,COVID,COVID/images/COVID-1442.png,COVID/masks/COVID-1442.png,True,G00480,train,2fef51e9d0a866673abc56c92a06cc44,812fc31b175f4cc6
COVID-1443.png,COVID,COVID/images/COVID-1443.png,COVID/masks/COVID-1443.png,True,G00481,train,98803b992692c9c82320198c313d0874,85780f070e1f5e1f
COVID-1444.png,COVID,COVID/images/COVID-1444.png,COVID/masks/COVID-1444.png,True,G00482,train,858b93dd11b1d5f37a52973866c26248,983866c38b4fea65
COVID-1445.png,COVID,COVID/images/COVID-1445.png,COVID/masks/COVID-1445.png,True,G00483,val,ce82d96646e4e946df7b888aac04a202,cc2ed3435643595d
COVID-1446.png,COVID,COVID/images/COVID-1446.png,COVID/masks/COVID-1446.png,True,G00484,train,bf5ef1428ff2563c523b383c2e326a86,8978061a5f4e5b5e
COVID-1447.png,COVID,COVID/images/COVID-1447.png,COVID/masks/COVID-1447.png,True,G00485,train,733ff7dbc13e41ba78ad5d5f1891ed7c,b83de4c65a1f4558
COVID-1448.png,COVID,COVID/images/COVID-1448.png,COVID/masks/COVID-1448.png,True,G00483,val,0572e77d60214e68e3c102b659441210,cc2ed3435643595d
COVID-1449.png,COVID,COVID/images/COVID-1449.png,COVID/masks/COVID-1449.png,True,G00336,train,a8e32a15ee2ba1a4f358be3a14ae113c,ec7d52534b4b4d0c
COVID-145.png,COVID,COVID/images/COVID-145.png,COVID/masks/COVID-145.png,True,G00486,train,13766f635aed65dea7fb80c5aa45da48,917f1618593ba8a7
COVID-1450.png,COVID,COVID/images/COVID-1450.png,COVID/masks/COVID-1450.png,True,G00487,train,643b61965f723dec3796ba852dbc538c,c765bb99192d24a6
COVID-1451.png,COVID,COVID/images/COVID-1451.png,COVID/masks/COVID-1451.png,True,G00488,train,315c15a4339da159de0d347c8ee6d138,cc264359471e4f4f
COVID-1452.png,COVID,COVID/images/COVID-1452.png,COVID/masks/COVID-1452.png,True,G00489,train,d331b759a2670d4521e14e01c86975a3,fd2cc7130f451c4d
COVID-1453.png,COVID,COVID/images/COVID-1453.png,COVID/masks/COVID-1453.png,True,G00490,train,784a711f1898160837a5fc38281676fe,f869c7965ed0981a
COVID-1454.png,COVID,COVID/images/COVID-1454.png,COVID/masks/COVID-1454.png,True,G00491,train,80118e99dbf72e1744a14b41374a0d5f,dc1a131e293cb5b5
COVID-1455.png,COVID,COVID/images/COVID-1455.png,COVID/masks/COVID-1455.png,True,G00492,train,446c4039288a6bb6b12217be7a795ad4,9879e51697c61745
COVID-1456.png,COVID,COVID/images/COVID-1456.png,COVID/masks/COVID-1456.png,True,G00493,train,825763c6d109f627f1edc19472033346,db31a5078e1eb07a
COVID-1457.png,COVID,COVID/images/COVID-1457.png,COVID/masks/COVID-1457.png,True,G00494,train,2c75359d50b04fb6fc2045fb7303449e,817f041e191f5f1d
COVID-1458.png,COVID,COVID/images/COVID-1458.png,COVID/masks/COVID-1458.png,True,G00495,val,6af1dd86a70e3068a320b66caedb2b19,d86c861b1e5a3e6c
COVID-1459.png,COVID,COVID/images/COVID-1459.png,COVID/masks/COVID-1459.png,True,G00496,val,71a80ea675553f9dc98f20d88c2696c0,b82de6921aadac6c
COVID-146.png,COVID,COVID/images/COVID-146.png,COVID/masks/COVID-146.png,True,G00497,val,2b87dda723daaf80bcdbb3c7e081eab7,d42d53434f4e2d2e
COVID-1460.png,COVID,COVID/images/COVID-1460.png,COVID/masks/COVID-1460.png,True,G00498,test,e7aea9e3249faa40c8768a34f272c58f,817b56061a5f5c7a
COVID-1461.png,COVID,COVID/images/COVID-1461.png,COVID/masks/COVID-1461.png,True,G00365,train,ad5a2a9a6962c4be264b9328e4495c5c,9979851f14561e3d
COVID-1462.png,COVID,COVID/images/COVID-1462.png,COVID/masks/COVID-1462.png,True,G00499,test,ffc5b0e1f37d6163fb9499efd2b117f9,d969861ec50eb14f
COVID-1463.png,COVID,COVID/images/COVID-1463.png,COVID/masks/COVID-1463.png,True,G00500,test,54f5e4cd8a54bc43d61829e23843a5dc,d43c5343f55a5343
COVID-1464.png,COVID,COVID/images/COVID-1464.png,COVID/masks/COVID-1464.png,True,G00501,train,f6a8237df865eea1957255d808bfae5b,c879a3e2b73c2925
COVID-1465.png,COVID,COVID/images/COVID-1465.png,COVID/masks/COVID-1465.png,True,G00502,train,c89f423ab472b300a5fb34390a223e21,c43d432d4ea53cb5
COVID-1466.png,COVID,COVID/images/COVID-1466.png,COVID/masks/COVID-1466.png,True,G00503,train,a6f6707e3f5fea84e29afeb2ff73f057,d57dc61a478e4287
COVID-1467.png,COVID,COVID/images/COVID-1467.png,COVID/masks/COVID-1467.png,True,G00504,train,bebf45ae92fc87e7f5e600443e6c8195,d91dc41e965a1e56
COVID-1468.png,COVID,COVID/images/COVID-1468.png,COVID/masks/COVID-1468.png,True,G00505,train,62b10f71df6cf53895f98bab6f0f3dc9,d43e07430d0d5f5d
COVID-1469.png,COVID,COVID/images/COVID-1469.png,COVID/masks/COVID-1469.png,True,G00506,train,01025d0e3c6fdb37bc4dc6d1b6ec87b6,9c75c51ac32b3b07
COVID-147.png,COVID,COVID/images/COVID-147.png,COVID/masks/COVID-147.png,True,G00507,train,5326a366a38dcc93ae1e03397f27bb8e,9d2f821b0b1f0b4f
COVID-1470.png,COVID,COVID/images/COVID-1470.png,COVID/masks/COVID-1470.png,True,G00508,train,67aa7a49a612fd4caed0f30b822816dc,dc78e2e2bbe92111
COVID-1471.png,COVID,COVID/images/COVID-1471.png,COVID/masks/COVID-1471.png,True,G00509,train,a950ac4fa4199c18040419cb8a008999,de2f430f4e4b0a2e
COVID-1472.png,COVID,COVID/images/COVID-1472.png,COVID/masks/COVID-1472.png,True,G00510,train,160bf3971c3e9713ad72d2f1cea40fe5,996dc71a1a451b5d
COVID-1473.png,COVID,COVID/images/COVID-1473.png,COVID/masks/COVID-1473.png,True,G00511,train,a2a1a1342cf308b8b157dc2503acf8b9,8d2dd21ac14f8fc3
COVID-1474.png,COVID,COVID/images/COVID-1474.png,COVID/masks/COVID-1474.png,True,G00512,train,674e8049557549f273559d941d635d19,9d35c2529b4b1f1a
COVID-1475.png,COVID,COVID/images/COVID-1475.png,COVID/masks/COVID-1475.png,True,G00513,train,0a6ff67fe264303a4d56c4746a80aa35,da69855e861aea4e
COVID-1476.png,COVID,COVID/images/COVID-1476.png,COVID/masks/COVID-1476.png,True,G00514,train,eac4ef7c3ddfc4b9e5bf838515589192,dd2cc24b55485f5c
COVID-1477.png,COVID,COVID/images/COVID-1477.png,COVID/masks/COVID-1477.png,True,G00515,val,fff8b012a2c891d0424671aed05b17e2,d949c71ec396160f
COVID-1478.png,COVID,COVID/images/COVID-1478.png,COVID/masks/COVID-1478.png,True,G00516,test,8947d159a6832b83a50a4e3477b646bc,990e9259f2f8e2b8
COVID-1479.png,COVID,COVID/images/COVID-1479.png,COVID/masks/COVID-1479.png,True,G00517,val,6d4f56a0b96e3375d1d15bb874aa7afa,9d45c65a471e5a0f
COVID-148.png,COVID,COVID/images/COVID-148.png,COVID/masks/COVID-148.png,True,G00518,test,fabc30726a5269e0f9c350ac87bd74dc,d96b12194932e5e7
COVID-1480.png,COVID,COVID/images/COVID-1480.png,COVID/masks/COVID-1480.png,True,G00519,val,1f78a9680529ff8c30382f03e93e07fd,882f061b0d7d4e6f
COVID-1481.png,COVID,COVID/images/COVID-1481.png,COVID/masks/COVID-1481.png,True,G00520,train,b9f3b0166fa6789cf79e7c255eea49a2,cc2f834b4d4c6c6d
COVID-1482.png,COVID,COVID/images/COVID-1482.png,COVID/masks/COVID-1482.png,True,G00521,test,756f6359c72dd8e9679d5926a3ba9882,8c2cf21f9f4f0d11
COVID-1483.png,COVID,COVID/images/COVID-1483.png,COVID/masks/COVID-1483.png,True,G00522,test,44e5c3cea78875e7499fc439142928b8,987b85d61e8a4d1e
COVID-1484.png,COVID,COVID/images/COVID-1484.png,COVID/masks/COVID-1484.png,True,G00523,train,ae970bb07f531364bed0e066d8da6321,d42ec34f474c4f4c
COVID-1485.png,COVID,COVID/images/COVID-1485.png,COVID/masks/COVID-1485.png,True,G00524,train,d5f6b44094b9207e66f5d19148c6e2a8,db75d046c65a9949
COVID-1486.png,COVID,COVID/images/COVID-1486.png,COVID/masks/COVID-1486.png,True,G00525,train,2baedf6651a17df9c0403cccefa985ac,997c5618497bbd82
COVID-1487.png,COVID,COVID/images/COVID-1487.png,COVID/masks/COVID-1487.png,True,G00526,train,3fc58f74a433ae258b89322587be91b3,cd6c8243d74e0f4e
COVID-1488.png,COVID,COVID/images/COVID-1488.png,COVID/masks/COVID-1488.png,True,G00527,val,5312a2f88f8a6bf3a5aaea49a0b18d91,ca2da51796569f50
COVID-1489.png,COVID,COVID/images/COVID-1489.png,COVID/masks/COVID-1489.png,True,G00528,train,ab2bdb23913442e6372df206acbea34c,817b445e1f7a592a
COVID-149.png,COVID,COVID/images/COVID-149.png,COVID/masks/COVID-149.png,True,G00529,test,8bed62eca3fdbf589e81f836c7b2f43c,c179121a5b1d1f4f
COVID-1490.png,COVID,COVID/images/COVID-1490.png,COVID/masks/COVID-1490.png,True,G00530,train,1857be53602dce194c6cde5cdbc645ec,897d165a1a5d5b0d
COVID-1491.png,COVID,COVID/images/COVID-1491.png,COVID/masks/COVID-1491.png,True,G00531,train,75f1ca0b8d472e1a6c7a2e1444436c0e,dd20c21fc63d395c
COVID-1492.png,COVID,COVID/images/COVID-1492.png,COVID/masks/COVID-1492.png,True,G00532,train,7c52eb02c084071131c2aa0199f57c5d,912a46d61be92bf4
COVID-1493.png,COVID,COVID/images/COVID-1493.png,COVID/masks/COVID-1493.png,True,G00533,train,624e04ebbc5db40ce5003306468fcd63,953e06561be568f4
COVID-1494.png,COVID,COVID/images/COVID-1494.png,COVID/masks/COVID-1494.png,True,G00534,train,e779a4ea68d8a92039b79f114e74f254,992dc79a9e131f05
COVID-1495.png,COVID,COVID/images/COVID-1495.png,COVID/masks/COVID-1495.png,True,G00535,train,a9e4a292a291dfe58eb8679432d5eda2,817d5e1a7b6560e2
COVID-1496.png,COVID,COVID/images/COVID-1496.png,COVID/masks/COVID-1496.png,True,G00536,train,f8d8d6fc589473e8583d9e7944b829d7,d979161e1e4918bd
COVID-1497.png,COVID,COVID/images/COVID-1497.png,COVID/masks/COVID-1497.png,True,G00537,train,59e3a96c647293373b22ebdcbc11a0da,d86195c6971aeb0d
COVID-1498.png,COVID,COVID/images/COVID-1498.png,COVID/masks/COVID-1498.png,True,G00538,train,4266532497938c11a3c0080341890121,817d0e1e5b595978
COVID-1499.png,COVID,COVID/images/COVID-1499.png,COVID/masks/COVID-1499.png,True,G00539,val,4c9d97bfaa11423e9f29ba8a3f1802f0,d438531f3c3036b7
COVID-15.png,COVID,COVID/images/COVID-15.png,COVID/masks/COVID-15.png,True,G00540,val,49cf4d6e32935cdccdabea90ff6ff444,dd2ad35f4c1c480f
COVID-150.png,COVID,COVID/images/COVID-150.png,COVID/masks/COVID-150.png,True,G00541,test,349a70af0748a3fa422e9e60fc2878ad,997b86161c477b1c
COVID-1500.png,COVID,COVID/images/COVID-1500.png,COVID/masks/COVID-1500.png,True,G00542,train,d0b1eafaa7c5489375d56d1c8962d582,d8596677b86a6161
COVID-1501.png,COVID,COVID/images/COVID-1501.png,COVID/masks/COVID-1501.png,True,G00543,train,4c8f08063f097ce0f88e99190468ecee,c43d434b0f0f6f0d
COVID-1502.png,COVID,COVID/images/COVID-1502.png,COVID/masks/COVID-1502.png,True,G00544,train,f0f1a1deb6a3cd6f130f877a35e5a871,813b5e96145a6b6d
COVID-1503.png,COVID,COVID/images/COVID-1503.png,COVID/masks/COVID-1503.png,True,G00545,train,7edee20094cac3168b39e5954758b5c4,cf2c94c2e373ab30
COVID-1504.png,COVID,COVID/images/COVID-1504.png,COVID/masks/COVID-1504.png,True,G00546,train,9b5483978da573767660b80fcdc887df,987b821a1b5a1e7e
COVID-1505.png,COVID,COVID/images/COVID-1505.png,COVID/masks/COVID-1505.png,True,G00547,train,64b4053ab3da694d27236d919792af73,9178175a1d7939ac
COVID-1506.png,COVID,COVID/images/COVID-1506.png,COVID/masks/COVID-1506.png,True,G00548,train,7aa7dbfbd919c0741cd0faa2eb3b30a6,9971c71e451e5a47
COVID-1507.png,COVID,COVID/images/COVID-1507.png,COVID/masks/COVID-1507.png,True,G00549,train,11ad4d19236c48ae955235628bcf7f05,d43dc2438a0fcb4f
COVID-1508.png,COVID,COVID/images/COVID-1508.png,COVID/masks/COVID-1508.png,True,G00550,train,3a4bd4384728c3b183be5ee63b67ae66,81695e5e1a154d7b
COVID-1509.png,COVID,COVID/images/COVID-1509.png,COVID/masks/COVID-1509.png,True,G00551,train,4e9401cdf35a3764b629f6d18b0aacc2,dc2de6d21b0d1e0e
COVID-151.png,COVID,COVID/images/COVID-151.png,COVID/masks/COVID-151.png,True,G00552,train,4fbef19ad2fe8c7835752ba3c5bed46a,997dc116d743125b
COVID-1510.png,COVID,COVID/images/COVID-1510.png,COVID/masks/COVID-1510.png,True,G00553,train,c1a9729df5ffc4b2d87bac6f56f381bf,8c2e035d4a7ceeaa
COVID-1511.png,COVID,COVID/images/COVID-1511.png,COVID/masks/COVID-1511.png,True,G00554,train,46e579f48bf66bc969b36e681bff1967,c149165e0f6d2d3d
COVID-1512.png,COVID,COVID/images/COVID-1512.png,COVID/masks/COVID-1512.png,True,G00555,val,93656ae557d99faa03ed642f4839b4de,bfe981479626d614
COVID-1513.png,COVID,COVID/images/COVID-1513.png,COVID/masks/COVID-1513.png,True,G00556,train,d76a24b867cfaa4638ecd5c7a9465b66,9948167be9f0e8e4
COVID-1514.png,COVID,COVID/images/COVID-1514.png,COVID/masks/COVID-1514.png,True,G00557,val,39dfb09294a80a23fe6aef56877ef5b3,d979c55616c2175c
COVID-1515.png,COVID,COVID/images/COVID-1515.png,COVID/masks/COVID-1515.png,True,G00558,test,017e8b31c2cbf0f645f68af6c4d4e2ad,892fe0e20f2f6c78
COVID-1516.png,COVID,COVID/images/COVID-1516.png,COVID/masks/COVID-1516.png,True,G00559,test,9efb6644ce70ea2700a1e9bd5c9ed83c,897b561a5e0a3b2d
COVID-1517.png,COVID,COVID/images/COVID-1517.png,COVID/masks/COVID-1517.png,True,G00560,test,a76801d8fd8f6020218cc6a85196a686,b925c61ac74a9fc1
COVID-1518.png,COVID,COVID/images/COVID-1518.png,COVID/masks/COVID-1518.png,True,G00561,val,3d8b03cacece492adba91acc830529cb,f87d65c2d71e1c08
COVID-1519.png,COVID,COVID/images/COVID-1519.png,COVID/masks/COVID-1519.png,True,G00562,train,198413f56682c12e36659afa0ef43859,952806171f0f5f5d
COVID-152.png,COVID,COVID/images/COVID-152.png,COVID/masks/COVID-152.png,True,G00563,train,daa35f520f8fc4c3f74968ff94b65587,dd0e43435b4d4d4e
COVID-1520.png,COVID,COVID/images/COVID-1520.png,COVID/masks/COVID-1520.png,True,G00436,val,8630beac5b8a0d12b431a5f8c0fd6df3,bd2cc21ac7c31bc5
COVID-1521.png,COVID,COVID/images/COVID-1521.png,COVID/masks/COVID-1521.png,True,G00564,train,66b882a967556024836a4cee583679a6,ca39b10686c6bed6
COVID-1522.png,COVID,COVID/images/COVID-1522.png,COVID/masks/COVID-1522.png,True,G00565,train,94a96c2bd2e0209260d90b4de35049a5,817e1618595f596d
COVID-1523.png,COVID,COVID/images/COVID-1523.png,COVID/masks/COVID-1523.png,True,G00566,val,c88510a87adcdfaba28418913bf2dae8,d52a77c1ea71e121
COVID-1524.png,COVID,COVID/images/COVID-1524.png,COVID/masks/COVID-1524.png,True,G00567,val,82a9281acf17ca6a6f0f8f1af15cde6b,d879d7e78a1ac812
COVID-1525.png,COVID,COVID/images/COVID-1525.png,COVID/masks/COVID-1525.png,True,G00568,test,b8967f8388dcd719239ab9941b3fb07d,d42e030f0c5e3e7e
COVID-1526.png,COVID,COVID/images/COVID-1526.png,COVID/masks/COVID-1526.png,True,G00569,train,01c82d394bd4dda7d4605e0b04cf04df,b967960c961e869e
COVID-1527.png,COVID,COVID/images/COVID-1527.png,COVID/masks/COVID-1527.png,True,G00570,test,23465235fe7ef1823c90cabf1d1a99ab,d879f516c7c24a4a
COVID-1528.png,COVID,COVID/images/COVID-1528.png,COVID/masks/COVID-1528.png,True,G00571,train,7e7460ba82c74fd2538646ce6c57124e,e96916161a1f703f
COVID-1529.png,COVID,COVID/images/COVID-1529.png,COVID/masks/COVID-1529.png,True,G00572,test,6c4552be0327ebd25992680e2ceaa327,99790596175e565a
COVID-153.png,COVID,COVID/images/COVID-153.png,COVID/masks/COVID-153.png,True,G00573,train,74b1eab5519c1e6c15d2660321328f29,cd3f20e03b4b49b9
COVID-1530.png,COVID,COVID/images/COVID-1530.png,COVID/masks/COVID-1530.png,True,G00574,train,7d4dd63fe74a62154acb67518d2cacb3,9869d61ec606cf4b
COVID-1531.png,COVID,COVID/images/COVID-1531.png,COVID/masks/COVID-1531.png,True,G00575,train,ff00f453029b6f11ad0bed4dad1abf13,eb69c696da1f8016
COVID-1532.png,COVID,COVID/images/COVID-1532.png,COVID/masks/COVID-1532.png,True,G00576,train,d63101b16793b56c4c78640e75554cf4,d879d61647535a49
COVID-1533.png,COVID,COVID/images/COVID-1533.png,COVID/masks/COVID-1533.png,True,G00577,train,436186eb15a37346243f6a87fb18bf95,d925c2afe21aa50f
COVID-1534.png,COVID,COVID/images/COVID-1534.png,COVID/masks/COVID-1534.png,True,G00578,train,39e37ee3d04f416ae32bce9bcb8d1f01,d56dc21a595f5868
COVID-1535.png,COVID,COVID/images/COVID-1535.png,COVID/masks/COVID-1535.png,True,G00579,train,3ab1238aff20d2c8fbf93c021f373b79,d969c416c75653c3
COVID-1536.png,COVID,COVID/images/COVID-1536.png,COVID/masks/COVID-1536.png,True,G00580,train,2591443e5329198eb4f13e53f7ab173d,dc38e286e53b8969
COVID-1537.png,COVID,COVID/images/COVID-1537.png,COVID/masks/COVID-1537.png,True,G00581,train,b8d11d0f06aaf62ee81c17ecb4b01458,8178061e1a7b7d79
COVID-1538.png,COVID,COVID/images/COVID-1538.png,COVID/masks/COVID-1538.png,True,G00582,val,0a83d09b5d56b5c43b86c5d80ace4433,d92d861e0e1e5e4d
COVID-1539.png,COVID,COVID/images/COVID-1539.png,COVID/masks/COVID-1539.png,True,G00583,train,db41684e2b2db5eba4ddaadfa8fde62b,992fc4102be339db
COVID-154.png,COVID,COVID/images/COVID-154.png,COVID/masks/COVID-154.png,True,G00584,train,b04b04760940d252b2a598095f35ac14,ce2dd9c94a4d4a5a
COVID-1540.png,COVID,COVID/images/COVID-1540.png,COVID/masks/COVID-1540.png,True,G00585,train,ebcb3dd6f0e71b969355d79ba6c73cd9,83637c9c709e7287
COVID-1541.png,COVID,COVID/images/COVID-1541.png,COVID/masks/COVID-1541.png,True,G00586,train,5d61d36349dcfca751a8445acda86d4a,c0bc2f83294d5e5b
COVID-1542.png,COVID,COVID/images/COVID-1542.png,COVID/masks/COVID-1542.png,True,G00587,train,3c085abceadcae32a289a7aabaeda9b8,9d24e20be72faf04
COVID-1543.png,COVID,COVID/images/COVID-1543.png,COVID/masks/COVID-1543.png,True,G00588,train,224350355421a165ee5675c3689c93a7,d42bd75a48435e6c
COVID-1544.png,COVID,COVID/images/COVID-1544.png,COVID/masks/COVID-1544.png,True,G00589,train,dd0ee7cf9ea5cbd8fc6760afe2abf943,d47f23822b0d4757
COVID-1545.png,COVID,COVID/images/COVID-1545.png,COVID/masks/COVID-1545.png,True,G00590,train,cb9da91341ce131381df4a285b7eaa2a,9919c7c68e9ac959
COVID-1546.png,COVID,COVID/images/COVID-1546.png,COVID/masks/COVID-1546.png,True,G00591,train,bcc62122ecb7f5dfc4f2bdab9fb4c593,906cc6931bcf1759
COVID-1547.png,COVID,COVID/images/COVID-1547.png,COVID/masks/COVID-1547.png,True,G00592,train,dab8b3ea8dcaee2dd8105111a989681d,9909d71e1b591a5b
COVID-1548.png,COVID,COVID/images/COVID-1548.png,COVID/masks/COVID-1548.png,True,G00593,train,113923b09af0bed2f16c7e19a121a24b,9965865ac7184f5e
COVID-1549.png,COVID,COVID/images/COVID-1549.png,COVID/masks/COVID-1549.png,True,G00594,train,7f0e565e07b437364d3ab09ba4f12d77,dd2ec21b9b41c31d
COVID-155.png,COVID,COVID/images/COVID-155.png,COVID/masks/COVID-155.png,True,G00595,train,dad2743b4d1f274312f0872175c8d2be,9d2f40d31b93431f
COVID-1550.png,COVID,COVID/images/COVID-1550.png,COVID/masks/COVID-1550.png,True,G00596,val,750e6150c385de82877fa010f7f3b14a,9d2d861a1f055f4d
COVID-1551.png,COVID,COVID/images/COVID-1551.png,COVID/masks/COVID-1551.png,True,G00597,val,5330c1419ab3e5ff7aa373dc8c84ecc1,d939e6761a686938
COVID-1552.png,COVID,COVID/images/COVID-1552.png,COVID/masks/COVID-1552.png,True,G00598,val,376538807cf4870fc5e24c23efc39660,807f4dcf1610df86
COVID-1553.png,COVID,COVID/images/COVID-1553.png,COVID/masks/COVID-1553.png,True,G00440,train,5c1827b5e4334e775a7f2a03000d9b2f,9c6da01ae33ccb87
COVID-1554.png,COVID,COVID/images/COVID-1554.png,COVID/masks/COVID-1554.png,True,G00599,test,e192d823d6d24a9cc83a9b63ba33b18c,986de1169ef0d80f
COVID-1555.png,COVID,COVID/images/COVID-1555.png,COVID/masks/COVID-1555.png,True,G00600,test,082a849a257baba11555e12824d6484c,8a2ec39b0d471e3e
COVID-1556.png,COVID,COVID/images/COVID-1556.png,COVID/masks/COVID-1556.png,True,G00601,test,771d2a3b5c72814deaa967f11e7dbecb,9b2aff00ff01941d
COVID-1557.png,COVID,COVID/images/COVID-1557.png,COVID/masks/COVID-1557.png,True,G00602,train,3865eb560ab751a48654132083ce7d6a,dd2a52531f564a1d
COVID-1558.png,COVID,COVID/images/COVID-1558.png,COVID/masks/COVID-1558.png,True,G00603,train,463f7423dbd8a92c0e78da51a26718a4,8d62c34f4a6a6b4b
COVID-1559.png,COVID,COVID/images/COVID-1559.png,COVID/masks/COVID-1559.png,True,G00604,train,b7a3741e9ec468bcf4243f6186b6cb37,8d60d30d645e595f
COVID-156.png,COVID,COVID/images/COVID-156.png,COVID/masks/COVID-156.png,True,G00605,train,2743bdcef206ed04c5eb7b2688d61172,d96d871f1e010f56
COVID-1560.png,COVID,COVID/images/COVID-1560.png,COVID/masks/COVID-1560.png,True,G00606,train,c3749bb6160cc50aaa37ed52768b3a35,cd55820ed249f8f3
COVID-1561.png,COVID,COVID/images/COVID-1561.png,COVID/masks/COVID-1561.png,True,G00607,train,e135a56a1cf82e1af43836955d806a4e,817f04ded8e9e9a0
COVID-1562.png,COVID,COVID/images/COVID-1562.png,COVID/masks/COVID-1562.png,True,G00608,train,9618e960dacc99863da202b48268d548,c92afd30b272e26a
COVID-1563.png,COVID,COVID/images/COVID-1563.png,COVID/masks/COVID-1563.png,True,G00609,train,a40d7d36400a93044c0e36ee7fd0794e,da11a1c7d6ce8c4e
COVID-1564.png,COVID,COVID/images/COVID-1564.png,COVID/masks/COVID-1564.png,True,G00610,train,8a293a2101176c2d72cb3910562dd6ef,d42e434bcf083f2d
COVID-1565.png,COVID,COVID/images/COVID-1565.png,COVID/masks/COVID-1565.png,True,G00611,train,49f1dfcf0d385ac3bf3c971e21f13bd7,9979659696163ec4
COVID-1566.png,COVID,COVID/images/COVID-1566.png,COVID/masks/COVID-1566.png,True,G00612,train,bdb9bddf431c10cdf8086e238eb1c12d,9f26a30fec2e8d60
COVID-1567.png,COVID,COVID/images/COVID-1567.png,COVID/masks/COVID-1567.png,True,G00613,train,28a44d043c8f9946fb5579cb360016bc,817b141eda5d6978
COVID-1568.png,COVID,COVID/images/COVID-1568.png,COVID/masks/COVID-1568.png,True,G00614,train,7275bc3603d8d8c614cc840426cc59b9,c03c3f032d4c5e7b
COVID-1569.png,COVID,COVID/images/COVID-1569.png,COVID/masks/COVID-1569.png,True,G00615,train,c2f0a3a134d296523641588d345f95d4,da29a54786969e56
COVID-157.png,COVID,COVID/images/COVID-157.png,COVID/masks/COVID-157.png,True,G00616,train,56589d69171497e79c68e222c5b02d5a,817b071e161b5f59
COVID-1570.png,COVID,COVID/images/COVID-1570.png,COVID/masks/COVID-1570.png,True,G00617,val,2dd55357053dcd8465d8d5d2214e827b,a97907969e829f1a
COVID-1571.png,COVID,COVID/images/COVID-1571.png,COVID/masks/COVID-1571.png,True,G00618,val,3cb0d5cfa642e267e1aa26e7de0d21c8,d42f22f22f41435f
COVID-1572.png,COVID,COVID/images/COVID-1572.png,COVID/masks/COVID-1572.png,True,G00619,val,6b0621dc7e7583ec4279cb0b2897a4e9,cb11d0879e1abe39
COVID-1573.png,COVID,COVID/images/COVID-1573.png,COVID/masks/COVID-1573.png,True,G00620,test,1e75b623dbfeca04172cdbdebe9972ec,887f041e5b4f5c55
COVID-1574.png,COVID,COVID/images/COVID-1574.png,COVID/masks/COVID-1574.png,True,G00621,test,55cd18ed26ad4789c01ef54697f45e9c,dc2b531d28f2c783
COVID-1575.png,COVID,COVID/images/COVID-1575.png,COVID/masks/COVID-1575.png,True,G00622,train,fb60a9c4a138aaa7102f6ac02d940eed,8b29950e961adb7a
COVID-1576.png,COVID,COVID/images/COVID-1576.png,COVID/masks/COVID-1576.png,True,G00623,test,1fdfc648079a3bb0523241a81bd43f5c,816d065b095f1b5f
COVID-1577.png,COVID,COVID/images/COVID-1577.png,COVID/masks/COVID-1577.png,True,G00624,train,eb219ed0cafbee74e03872430f6779a0,997dc0165f431a5b
COVID-1578.png,COVID,COVID/images/COVID-1578.png,COVID/masks/COVID-1578.png,True,G00625,train,31545a33777b21ae5088361c7b142bbb,cd6cf4705a72f052
COVID-1579.png,COVID,COVID/images/COVID-1579.png,COVID/masks/COVID-1579.png,True,G00626,train,6a72ccd181d377e6a1bca1d07db92f09,817f965a58575919
COVID-158.png,COVID,COVID/images/COVID-158.png,COVID/masks/COVID-158.png,True,G00627,train,ed2ac741134fd31097fe47cc2af85f13,d83ac3879ec6523a
COVID-1580.png,COVID,COVID/images/COVID-1580.png,COVID/masks/COVID-1580.png,True,G00628,train,4b926090de9ad7a79e3dc31b3d229883,9969c51ac71e9e49
COVID-1581.png,COVID,COVID/images/COVID-1581.png,COVID/masks/COVID-1581.png,True,G00629,train,83579421bc2b4cefc51e46e65910ff3c,9d6de096c783930b
COVID-1582.png,COVID,COVID/images/COVID-1582.png,COVID/masks/COVID-1582.png,True,G00630,train,da7e0136ec2ba366475e0391b7d71fc8,883fc74a1f4e1e1c
COVID-1583.png,COVID,COVID/images/COVID-1583.png,COVID/masks/COVID-1583.png,True,G00631,val,dc76cb9e40b3d431d1f8b03a5afb3879,b82d471a1e6d2d3c
COVID-1584.png,COVID,COVID/images/COVID-1584.png,COVID/masks/COVID-1584.png,True,G00632,test,d759fbda4f3581f217a63194f3ba7b93,de2dd01a93431f4b
COVID-1585.png,COVID,COVID/images/COVID-1585.png,COVID/masks/COVID-1585.png,True,G00633,train,fb6c891283f869cf362b085a6bbe4caa,dd2cd0c25b4d4e5a
COVID-1586.png,COVID,COVID/images/COVID-1586.png,COVID/masks/COVID-1586.png,True,G00634,train,0756b51406a2acb0f8ddde333aecbd1f,897dc65a194d5d1c
COVID-1587.png,COVID,COVID/images/COVID-1587.png,COVID/masks/COVID-1587.png,True,G00635,train,0be1ccc226b6141fe9e7ef429899dcfa,9d64c23e693dca29
COVID-1588.png,COVID,COVID/images/COVID-1588.png,COVID/masks/COVID-1588.png,True,G00636,train,439957463830f0e2f108829b36f35a82,817987c69a97c61e
COVID-1589.png,COVID,COVID/images/COVID-1589.png,COVID/masks/COVID-1589.png,True,G00637,train,e93e403772cb73401e2e8be587cbc146,8c2cd2d34f4d4c5d
COVID-159.png,COVID,COVID/images/COVID-159.png,COVID/masks/COVID-159.png,True,G00638,train,00b09884b96575eab6629bb413f05bef,cd2ad2c34f5d4c4c
COVID-1590.png,COVID,COVID/images/COVID-1590.png,COVID/masks/COVID-1590.png,True,G00639,test,2a64da0969dc08b102286ed5c7410dd0,d839a5c65e6b5119
COVID-1591.png,COVID,COVID/images/COVID-1591.png,COVID/masks/COVID-1591.png,True,G00640,val,0773aa4716b37c0a07f10be68744f32a,896f061a4d65deca
COVID-1592.png,COVID,COVID/images/COVID-1592.png,COVID/masks/COVID-1592.png,True,G00641,val,ab7fb8ddfdcf91b5146ee26d5466a05e,dd3fc212124f1a5d
COVID-1593.png,COVID,COVID/images/COVID-1593.png,COVID/masks/COVID-1593.png,True,G00642,train,5668f8a63ff75c4c4e5ad47f4b195a9d,d93d86d65a0c4d4b
COVID-1594.png,COVID,COVID/images/COVID-1594.png,COVID/masks/COVID-1594.png,True,G00643,test,c60aa2782d4e678b5531d12624bc5024,993d061a196f787c
COVID-1595.png,COVID,COVID/images/COVID-1595.png,COVID/masks/COVID-1595.png,True,G00644,train,d7a9374f6650e547141f92318124b4f9,912ace1bc78fa1a3
COVID-1596.png,COVID,COVID/images/COVID-1596.png,COVID/masks/COVID-1596.png,True,G00645,train,4f3ce11e62c63debb767971b1ce9f807,d811c74eca7af843
COVID-1597.png,COVID,COVID/images/COVID-1597.png,COVID/masks/COVID-1597.png,True,G00646,train,d97d6c540cb7c10942a10e6cd6d84ec6,dc2ec2c35f4f421a
COVID-1598.png,COVID,COVID/images/COVID-1598.png,COVID/masks/COVID-1598.png,True,G00647,test,8b13c344c79b8783e3ba367cfc0fd4c9,dd28520b97431f6b
COVID-1599.png,COVID,COVID/images/COVID-1599.png,COVID/masks/COVID-1599.png,True,G00648,train,1a7a8fd5047cc3329b5713acae7e392e,d879d65654571a19
COVID-16.png,COVID,COVID/images/COVID-16.png,COVID/masks/COVID-16.png,True,G00649,test,6e0534b84048705b7c72ba762fe31ddd,d53ec25f5c442c9c
COVID-160.png,COVID,COVID/images/COVID-160.png,COVID/masks/COVID-160.png,True,G00112,train,9784813ed917978581430bc645763553,9c3c420f7f483d74
COVID-1600.png,COVID,COVID/images/COVID-1600.png,COVID/masks/COVID-1600.png,True,G00650,train,63f88a52d2ffbc5ee95cfcd6eeb04b7e,d727484a0e672b2f
COVID-1601.png,COVID,COVID/images/COVID-1601.png,COVID/masks/COVID-1601.png,True,G00651,train,3b16f4c6981f02c2a2d024ce957ddb8b,886fc35b054f164f
COVID-1602.png,COVID,COVID/images/COVID-1602.png,COVID/masks/COVID-1602.png,True,G00652,test,41b652fcd9be31bca68eef9e9637cf3b,ca28a196d68fd393
COVID-1603.png,COVID,COVID/images/COVID-1603.png,COVID/masks/COVID-1603.png,True,G00653,val,1194e653ec521568d968b95035af7a08,d03d5383570f6e45
COVID-1604.png,COVID,COVID/images/COVID-1604.png,COVID/masks/COVID-1604.png,True,G00654,train,1c6a9e2e98886def4985d6e8fdeee305,d9f5c61e840e930f
COVID-1605.png,COVID,COVID/images/COVID-1605.png,COVID/masks/COVID-1605.png,True,G00655,train,5f1f389f28d1a4bdd7a6dafbda510395,dd0dc4561f455e19
COVID-1606.png,COVID,COVID/images/COVID-1606.png,COVID/masks/COVID-1606.png,True,G00656,train,c6d8109af83861c93a816dbddf221bb5,d94d471e184d1e1f
COVID-1607.png,COVID,COVID/images/COVID-1607.png,COVID/masks/COVID-1607.png,True,G00657,train,913959f5065747c98eeb625e84aae02c,887d0717165e736a
COVID-1608.png,COVID,COVID/images/COVID-1608.png,COVID/masks/COVID-1608.png,True,G00658,train,8ec32d277c5ac91ce7ef1b06b21fe9ef,bc6cc51ac31f1ac3
COVID-1609.png,COVID,COVID/images/COVID-1609.png,COVID/masks/COVID-1609.png,True,G00659,train,c17e7894d5c8aa2180eb0fe668f15f35,c1fed650584d7878
COVID-161.png,COVID,COVID/images/COVID-161.png,COVID/masks/COVID-161.png,True,G00660,train,b4a16bf40f94e6552f2c549ee42af62c,990c06161b5f5fce
COVID-1610.png,COVID,COVID/images/COVID-1610.png,COVID/masks/COVID-1610.png,True,G00661,val,e47a81dae3b13b681634c67242873ef5,9968c61ae33dea85
COVID-1611.png,COVID,COVID/images/COVID-1611.png,COVID/masks/COVID-1611.png,True,G00662,val,27a1d7ac9c09eb0892dad3539484bbc5,81b956865e945ed3
COVID-1612.png,COVID,COVID/images/COVID-1612.png,COVID/masks/COVID-1612.png,True,G00663,train,506865ebafda045c056240631b9f11a1,bc2c82928bafac3e
COVID-1613.png,COVID,COVID/images/COVID-1613.png,COVID/masks/COVID-1613.png,True,G00664,train,e496839daab4036814dbf562bdc6389a,9b601c161e3e733b
COVID-1614.png,COVID,COVID/images/COVID-1614.png,COVID/masks/COVID-1614.png,True,G00665,train,0875a89fe8b85453dc5fd0640d65e51f,9d6c961ae13dcc29
COVID-1615.png,COVID,COVID/images/COVID-1615.png,COVID/masks/COVID-1615.png,True,G00666,train,52478001826d2938f0d4a4f78cd17a82,897d061a1c1f3b1f
COVID-1616.png,COVID,COVID/images/COVID-1616.png,COVID/masks/COVID-1616.png,True,G00667,train,d72460b2127d6fbffe9546990ea6edfa,c84dc65a5f1b1b45
COVID-1617.png,COVID,COVID/images/COVID-1617.png,COVID/masks/COVID-1617.png,True,G00668,train,083d92f179eb332ffb2af063b096da8d,8179e7c612144f77
COVID-1618.png,COVID,COVID/images/COVID-1618.png,COVID/masks/COVID-1618.png,True,G00669,train,47f19112d6b4f28cb2d89598324a1a97,cd32c61f1b095d78
COVID-1619.png,COVID,COVID/images/COVID-1619.png,COVID/masks/COVID-1619.png,True,G00670,train,8ee7dc0e58ab6bf1c9623d3147e39c8e,c95f465a1c4b1b1e
COVID-162.png,COVID,COVID/images/COVID-162.png,COVID/masks/COVID-162.png,True,G00671,train,aedcec60a7d9227958f0f298c3bf5e45,a1ff845a0b390f33
COVID-1620.png,COVID,COVID/images/COVID-1620.png,COVID/masks/COVID-1620.png,True,G00672,train,a552ec3faff8dfc3027d151120a24ac6,9945165b075a4f5e
COVID-1621.png,COVID,COVID/images/COVID-1621.png,COVID/masks/COVID-1621.png,True,G00673,train,b796db034ad56b2599ea2b752bdbc34f,81690496585d7f7e
COVID-1622.png,COVID,COVID/images/COVID-1622.png,COVID/masks/COVID-1622.png,True,G00674,test,2b5abc38a621c1ea129ef0e2d7b79da2,f879c587c7c21a16
COVID-1623.png,COVID,COVID/images/COVID-1623.png,COVID/masks/COVID-1623.png,True,G00675,train,768244496edcc2dc98d3a570183f81ad,9d6c86131f171b1b
COVID-1624.png,COVID,COVID/images/COVID-1624.png,COVID/masks/COVID-1624.png,True,G00676,train,e480afa4505a5b1a4dc9bb823609d27c,dc48e2c2b8e9e1e9
COVID-1625.png,COVID,COVID/images/COVID-1625.png,COVID/masks/COVID-1625.png,True,G00677,val,f7697b1affcccbaba21a9b9f1c222d6f,c43d424b5f6e5b05
COVID-1626.png,COVID,COVID/images/COVID-1626.png,COVID/masks/COVID-1626.png,True,G00678,train,e34ff29726327645f1dda73dd869c0da,9d2dc212cfd71a89
COVID-1627.png,COVID,COVID/images/COVID-1627.png,COVID/masks/COVID-1627.png,True,G00679,train,8d5d08a1ed03ecc9deb63b5e5a516480,c63d410f4b2c2e6f
COVID-1628.png,COVID,COVID/images/COVID-1628.png,COVID/masks/COVID-1628.png,True,G00680,val,b7c85f4564fe376bd034eebc680293e5,da21a1468e5e9a5f
COVID-1629.png,COVID,COVID/images/COVID-1629.png,COVID/masks/COVID-1629.png,True,G00681,test,5d7e3c28f28d7dba73edf5ec4541c956,817f045a195d555f
COVID-163.png,COVID,COVID/images/COVID-163.png,COVID/masks/COVID-163.png,True,G00682,val,fd8bdaa2b74c1cd864a375f1264d195e,9b65c41ae60ebf42
COVID-1630.png,COVID,COVID/images/COVID-1630.png,COVID/masks/COVID-1630.png,True,G00683,test,0c7f70fb4e5e704beddb57cdf1f2f607,b929a6c69678686d
COVID-1631.png,COVID,COVID/images/COVID-1631.png,COVID/masks/COVID-1631.png,True,G00684,train,74fc67b25340eed18a487bc796e4dac0,817fe61a19451f1e
COVID-1632.png,COVID,COVID/images/COVID-1632.png,COVID/masks/COVID-1632.png,True,G00685,train,1dd95eff977a11eedb54c3fea10405ec,cd5eb043c3cb8719
COVID-1633.png,COVID,COVID/images/COVID-1633.png,COVID/masks/COVID-1633.png,True,G00686,train,60516df713f85b4d63417c72a7db4818,dd2cc75b1d4c581c
COVID-1634.png,COVID,COVID/images/COVID-1634.png,COVID/masks/COVID-1634.png,True,G00687,train,5cd0e5930d5b652c26fdfb70645926ec,c839b517b6b68a98
COVID-1635.png,COVID,COVID/images/COVID-1635.png,COVID/masks/COVID-1635.png,True,G00688,val,bad0f8278866aa2a9a97a9f4e9224021,d95b1618191f1aaf
COVID-1636.png,COVID,COVID/images/COVID-1636.png,COVID/masks/COVID-1636.png,True,G00689,test,4c21c6f90ae5ed8b8c20587089ea5ad0,9d64921ba93cca3d
COVID-1637.png,COVID,COVID/images/COVID-1637.png,COVID/masks/COVID-1637.png,True,G00690,train,e82eff763e775dba91d88b51d57db10e,d95bd2585d1a1e16
COVID-1638.png,COVID,COVID/images/COVID-1638.png,COVID/masks/COVID-1638.png,True,G00691,train,28321b885a672d6854ffbd9e9befc154,8d4cc34b4f4d1e4d
COVID-1639.png,COVID,COVID/images/COVID-1639.png,COVID/masks/COVID-1639.png,True,G00692,train,753cb0a2e4fb1aa118c8bd73872d2df0,cd6e465e68e6e0e0
COVID-164.png,COVID,COVID/images/COVID-164.png,COVID/masks/COVID-164.png,True,G00693,train,927dd77ef9d3d2906595ab0b678e866f,f97bc65a59491911
COVID-1640.png,COVID,COVID/images/COVID-1640.png,COVID/masks/COVID-1640.png,True,G00694,train,8630c68e8ca1c8091a7fdb7354cae96f,cd7dc4064f171b0b
COVID-1641.png,COVID,COVID/images/COVID-1641.png,COVID/masks/COVID-1641.png,True,G00695,train,8a5ae05d17e0f94ac211dca80ce4242f,d42f52531a1c5b1f
COVID-1642.png,COVID,COVID/images/COVID-1642.png,COVID/masks/COVID-1642.png,True,G00696,train,7db5db359d54af4fe238257b10e645fd,c93bc71e1e4d181d
COVID-1643.png,COVID,COVID/images/COVID-1643.png,COVID/masks/COVID-1643.png,True,G00697,val,f538bc9c12e2581971d8d6f3559abad1,d42e474f0c4c6e6e
COVID-1644.png,COVID,COVID/images/COVID-1644.png,COVID/masks/COVID-1644.png,True,G00698,train,a6965b7c6935fb717f700db73b8f609e,9c3ee1e79fc48484
COVID-1645.png,COVID,COVID/images/COVID-1645.png,COVID/masks/COVID-1645.png,True,G00699,train,f7d2dc3c3c5d31d7a488ba8d3e4eeb0e,cd2c561a486feaf0
COVID-1646.png,COVID,COVID/images/COVID-1646.png,COVID/masks/COVID-1646.png,True,G00700,train,5566c31782978de9e71957003a7dd226,c90986561ede5a5e
COVID-1647.png,COVID,COVID/images/COVID-1647.png,COVID/masks/COVID-1647.png,True,G00701,val,1caeb0f6e3554cf9030cb23ff40cf8a3,d16b06061e195f5f
COVID-1648.png,COVID,COVID/images/COVID-1648.png,COVID/masks/COVID-1648.png,True,G00702,test,91359e76aee644d5c6f5efc66e404164,9d618716e33dea09
COVID-1649.png,COVID,COVID/images/COVID-1649.png,COVID/masks/COVID-1649.png,True,G00703,test,938d8a594411cb2e22a36ab8e871b219,d03b92091c7f3b39
COVID-165.png,COVID,COVID/images/COVID-165.png,COVID/masks/COVID-165.png,True,G00704,train,3a47555c358a3c4e309a1475776e848b,cc2fd30d455c4f43
COVID-1650.png,COVID,COVID/images/COVID-1650.png,COVID/masks/COVID-1650.png,True,G00705,train,58d3467ac2113899ee379c821658d55f,d911841e87eabad6
COVID-1651.png,COVID,COVID/images/COVID-1651.png,COVID/masks/COVID-1651.png,True,G00706,train,59bb237a49a0e9a97b16569d59df89bc,d42f035c4c6e6c6e
COVID-1652.png,COVID,COVID/images/COVID-1652.png,COVID/masks/COVID-1652.png,True,G00707,train,34f95933f281bba2ba7060e65290e22d,d81ba7c67e784941
COVID-1653.png,COVID,COVID/images/COVID-1653.png,COVID/masks/COVID-1653.png,True,G00708,train,1fcbda6855d97745c1344a1f7c1f7343,817a161e1d5f5a39
COVID-1654.png,COVID,COVID/images/COVID-1654.png,COVID/masks/COVID-1654.png,True,G00709,train,85d6f55c0cacc73c6ce16983670f2bab,d0197fc6ee9e8848
COVID-1655.png,COVID,COVID/images/COVID-1655.png,COVID/masks/COVID-1655.png,True,G00710,train,9d1bd2d448c1cb3a3f8121280b8a9cb1,d03b53183a3b3aba
COVID-1656.png,COVID,COVID/images/COVID-1656.png,COVID/masks/COVID-1656.png,True,G00711,train,a9c5ef5302f9c59ed1564b51c7190b63,cd7c828ff2092d4d
COVID-1657.png,COVID,COVID/images/COVID-1657.png,COVID/masks/COVID-1657.png,True,G00712,train,3d6e9e62bccc69bd3f7ab8aafd414947,dd28d31fd2525b60
COVID-1658.png,COVID,COVID/images/COVID-1658.png,COVID/masks/COVID-1658.png,True,G00713,train,6396a4b5c2f46817351f8b83579d1331,9c6cc2b38b83398f
COVID-1659.png,COVID,COVID/images/COVID-1659.png,COVID/masks/COVID-1659.png,True,G00714,train,6686c79b694fcbffe5a4e5bd40306c57,d96a461a584c7d7c
COVID-166.png,COVID,COVID/images/COVID-166.png,COVID/masks/COVID-166.png,True,G00715,train,1b1ac772e2ecdd0c70182355f652cd4d,c80d871e471f4bcb
COVID-1660.png,COVID,COVID/images/COVID-1660.png,COVID/masks/COVID-1660.png,True,G00716,test,685fc336dbf34c6bd6d5ddb8dbfa9158,8f64d20f471f484f
COVID-1661.png,COVID,COVID/images/COVID-1661.png,COVID/masks/COVID-1661.png,True,G00717,val,592d18bb969b073399a3535a5c9e4504,dd6c934aa51ac31d
COVID-1662.png,COVID,COVID/images/COVID-1662.png,COVID/masks/COVID-1662.png,True,G00718,train,789cf8ff60b97a1d0abc3f1fdcd0a0c2,c9f9d61e924e1a1a
COVID-1663.png,COVID,COVID/images/COVID-1663.png,COVID/masks/COVID-1663.png,True,G00719,val,65815c7fc7be45904cf192797b43c774,836b849c9c8d35d7
COVID-1664.png,COVID,COVID/images/COVID-1664.png,COVID/masks/COVID-1664.png,True,G00720,test,c2b7ceb78c9b47f99a89f326692e8f74,987d0716164f395b
COVID-1665.png,COVID,COVID/images/COVID-1665.png,COVID/masks/COVID-1665.png,True,G00721,train,807498adae9ac77c5eba65a81316326f,807bc45c5a7a7ab8
COVID-1666.png,COVID,COVID/images/COVID-1666.png,COVID/masks/COVID-1666.png,True,G00722,val,653cccdca5bdc456d3a474ce53fea250,887fc706124f5f13
COVID-1667.png,COVID,COVID/images/COVID-1667.png,COVID/masks/COVID-1667.png,True,G00723,test,11285e2533998a712c4ea2c914bc8fd0,995fc61a4869eae2
COVID-1668.png,COVID,COVID/images/COVID-1668.png,COVID/masks/COVID-1668.png,True,G00724,train,cb2ae9cd12d82dad0ae26591e7048872,fd20d21fd24a1f4a
COVID-1669.png,COVID,COVID/images/COVID-1669.png,COVID/masks/COVID-1669.png,True,G00725,train,f9296358bfd9df701796c3ee9ea13ed0,913bc7c656505b59
COVID-167.png,COVID,COVID/images/COVID-167.png,COVID/masks/COVID-167.png,True,G00726,val,56f2520cab4d15125b6228670ee34314,913be7d61e44cf10
COVID-1670.png,COVID,COVID/images/COVID-1670.png,COVID/masks/COVID-1670.png,True,G00727,train,7d3188efe93a3a721e65ae7f497d8149,d82d871e1e1f1b0e
COVID-1671.png,COVID,COVID/images/COVID-1671.png,COVID/masks/COVID-1671.png,True,G00728,test,46ecd4384aecd526f1cd4fa467dba814,dd0ac34f4c286f4b
COVID-1672.png,COVID,COVID/images/COVID-1672.png,COVID/masks/COVID-1672.png,True,G00729,train,c7129092ea0b1860e3e3dc2886931d32,d81087665e5e5753
COVID-1673.png,COVID,COVID/images/COVID-1673.png,COVID/masks/COVID-1673.png,True,G00730,test,f302a31c3903c7d285f3d15defbccc28,843e17566a55aa57
COVID-1674.png,COVID,COVID/images/COVID-1674.png,COVID/masks/COVID-1674.png,True,G00731,train,b87e52a93684f8428f50d637c4341a5d,9e64e092cb3d847e
COVID-1675.png,COVID,COVID/images/COVID-1675.png,COVID/masks/COVID-1675.png,True,G00732,train,4af3c54c87d220540964c0dcdadf9c6d,c46e174a6d5d4955
COVID-1676.png,COVID,COVID/images/COVID-1676.png,COVID/masks/COVID-1676.png,True,G00733,train,7efc9017f4d239edc7a322bfb2b3b664,9d6de19652c35e90
COVID-1677.png,COVID,COVID/images/COVID-1677.png,COVID/masks/COVID-1677.png,True,G00734,train,745a16a746bbdcacf29f8c7971f7ec19,dd4cc2431f494d1f
COVID-1678.png,COVID,COVID/images/COVID-1678.png,COVID/masks/COVID-1678.png,True,G00735,train,e659b3409f8d297e84fc31a320fc194c,986dc61e95515d4e
COVID-1679.png,COVID,COVID/images/COVID-1679.png,COVID/masks/COVID-1679.png,True,G00736,train,3cc2e59b0216d29728616699b5299b5d,8d26830dc69ee7b2
COVID-168.png,COVID,COVID/images/COVID-168.png,COVID/masks/COVID-168.png,True,G00737,train,a046aa9ba6359f948ffcbbc5091077f7,d869c45e595c5e49
COVID-1680.png,COVID,COVID/images/COVID-1680.png,COVID/masks/COVID-1680.png,True,G00738,train,d6f9ffe621a9a247d039ad7d55f041b6,9a29975eb61aa949
COVID-1681.png,COVID,COVID/images/COVID-1681.png,COVID/masks/COVID-1681.png,True,G00739,train,1e4b9644424b7d615a78550d89b0a190,957987461e1f434b
COVID-1682.png,COVID,COVID/images/COVID-1682.png,COVID/masks/COVID-1682.png,True,G00740,train,4bfd10d98f240e6fe7bb5062b26f5007,cf2c9007d7539b0d
COVID-1683.png,COVID,COVID/images/COVID-1683.png,COVID/masks/COVID-1683.png,True,G00741,val,3b9cb95aa2f2c1abffe15e45555af339,95e90616155f5e1b
COVID-1684.png,COVID,COVID/images/COVID-1684.png,COVID/masks/COVID-1684.png,True,G00742,val,0e37e76b484b7249044ecf46a3a2d9ba,c87f851e125b5959
COVID-1685.png,COVID,COVID/images/COVID-1685.png,COVID/masks/COVID-1685.png,True,G00743,test,9cfc64a8c7f35473deace9e5199dab91,d42d471f4c2c6e4e
COVID-1686.png,COVID,COVID/images/COVID-1686.png,COVID/masks/COVID-1686.png,True,G00744,train,2440fa05a3a85c6ea8cf2ba809dee90b,cf3df0c3a7920b0c
COVID-1687.png,COVID,COVID/images/COVID-1687.png,COVID/masks/COVID-1687.png,True,G00745,train,2dee1c42b75137cc77fe65a2b93b8e55,897f071e174d1e51
COVID-1688.png,COVID,COVID/images/COVID-1688.png,COVID/masks/COVID-1688.png,True,G00746,train,1096dbfa2a06d31135cd026e7b8fcf74,8a2d95479e5a1d59
COVID-1689.png,COVID,COVID/images/COVID-1689.png,COVID/masks/COVID-1689.png,True,G00747,train,8d099171a7014167cf21abe5c5ed7d9b,d52f07d70a5d0f18
COVID-169.png,COVID,COVID/images/COVID-169.png,COVID/masks/COVID-169.png,True,G00748,train,2b23c7c74424cecca1ee50c423d56e2d,9979c716015f185f
COVID-1690.png,COVID,COVID/images/COVID-1690.png,COVID/masks/COVID-1690.png,True,G00749,train,c6a7d87b9480fa2ea25a69df1100eb1c,dd2de04207475e5d
COVID-1691.png,COVID,COVID/images/COVID-1691.png,COVID/masks/COVID-1691.png,True,G00750,train,6c421ac81948003b42641cc633c16442,dc2ed71b470a194b
COVID-1692.png,COVID,COVID/images/COVID-1692.png,COVID/masks/COVID-1692.png,True,G00751,train,f3b1a934bbfcc12876fe9e04826ec476,dd68d617454e1d43
COVID-1693.png,COVID,COVID/images/COVID-1693.png,COVID/masks/COVID-1693.png,True,G00752,train,9938042c70aee254c25fcc23a678ade9,817be40419597f79
COVID-1694.png,COVID,COVID/images/COVID-1694.png,COVID/masks/COVID-1694.png,True,G00753,train,fe2cc7cf129a47a6e0ea7153e714465b,b979c4160ece581f
COVID-1695.png,COVID,COVID/images/COVID-1695.png,COVID/masks/COVID-1695.png,True,G00754,train,7dee48ed959bf3d53462680170e5ba97,947e06571f19393c
COVID-1696.png,COVID,COVID/images/COVID-1696.png,COVID/masks/COVID-1696.png,True,G00755,train,4c03ec004f80167fad328595b0075146,9c3d46521b175b1d
COVID-1697.png,COVID,COVID/images/COVID-1697.png,COVID/masks/COVID-1697.png,True,G00756,train,06e1da62dd2f4ef6673aa4fd11edf6f4,997986165c4f195d
COVID-1698.png,COVID,COVID/images/COVID-1698.png,COVID/masks/COVID-1698.png,True,G00757,train,88874426192ef14fe829c3883b643436,bc1de7322b29213d
COVID-1699.png,COVID,COVID/images/COVID-1699.png,COVID/masks/COVID-1699.png,True,G00758,val,d04437522bd46cdab58f404287a65613,903847475f5a7879
COVID-17.png,COVID,COVID/images/COVID-17.png,COVID/masks/COVID-17.png,True,G00759,val,ad723892b73506714660bf9ae4bd8729,d82fc61e8ccc1dc6
COVID-170.png,COVID,COVID/images/COVID-170.png,COVID/masks/COVID-170.png,True,G00760,test,b2e2b4b47756ca38e1fe75316bc81939,907f861a584d395f
COVID-1700.png,COVID,COVID/images/COVID-1700.png,COVID/masks/COVID-1700.png,True,G00761,test,4f0d5fe5912d428234d6ecab2a2a7735,887f875e18c65e1c
COVID-1701.png,COVID,COVID/images/COVID-1701.png,COVID/masks/COVID-1701.png,True,G00762,val,db198f265c5bd40a6ea31cf0345eb2da,dd2ac3c79e1c2878
COVID-1702.png,COVID,COVID/images/COVID-1702.png,COVID/masks/COVID-1702.png,True,G00763,test,781d760eb13950d36c2e2d48d19b6b1f,897f86121f0f1e1b
COVID-1703.png,COVID,COVID/images/COVID-1703.png,COVID/masks/COVID-1703.png,True,G00764,val,33fb4fd1039978aa4659e5bf43cb6ae6,a97f86065e045f1b
COVID-1704.png,COVID,COVID/images/COVID-1704.png,COVID/masks/COVID-1704.png,True,G00765,train,ff1cd7c077a99c0b6a084396c7fc4f41,f979f6868aa8381a
COVID-1705.png,COVID,COVID/images/COVID-1705.png,COVID/masks/COVID-1705.png,True,G00766,train,c2af6c90fb5f3059bd443422c42d4d98,912bc65e7a466939
COVID-1706.png,COVID,COVID/images/COVID-1706.png,COVID/masks/COVID-1706.png,True,G00767,train,42036a0c1edb3ae4fe357bbdc315edb0,9879e71652c71ac3
COVID-1707.png,COVID,COVID/images/COVID-1707.png,COVID/masks/COVID-1707.png,True,G00768,test,fc6e5cc18d46c60d7377b6535a243e37,814e0e584d7dc2fb
COVID-1708.png,COVID,COVID/images/COVID-1708.png,COVID/masks/COVID-1708.png,True,G00769,train,4e83a112d4bbb6db13bca96c2a7d5ad1,896f865afa478361
COVID-1709.png,COVID,COVID/images/COVID-1709.png,COVID/masks/COVID-1709.png,True,G00770,train,8bea53a4408aa2e108f667daf5adda7a,813945465e3a5e5f
COVID-171.png,COVID,COVID/images/COVID-171.png,COVID/masks/COVID-171.png,True,G00771,train,ca7a6c829bb86f9dda9c7e072c691d31,dd2a530f1c6b3722
COVID-1710.png,COVID,COVID/images/COVID-1710.png,COVID/masks/COVID-1710.png,True,G00772,train,7e33259eb79d9757a6be4faea3eec0cb,907bc75e5655584a
COVID-1711.png,COVID,COVID/images/COVID-1711.png,COVID/masks/COVID-1711.png,True,G00773,train,2219edac1f557f62c8cfa9d2da01c8f5,906e86195dec78ea
COVID-1712.png,COVID,COVID/images/COVID-1712.png,COVID/masks/COVID-1712.png,True,G00774,train,e1e4b5024128ddac45ba756869ab5afb,917b5a5a5a1b1b0e
COVID-1713.png,COVID,COVID/images/COVID-1713.png,COVID/masks/COVID-1713.png,True,G00775,train,26e1a989a0a4d1439df760ba006f1f3a,915b1e185b39e8e6
COVID-1714.png,COVID,COVID/images/COVID-1714.png,COVID/masks/COVID-1714.png,True,G00776,train,9917e0d218357398ff50fb0c30df4341,b849c296175b4b1f
COVID-1715.png,COVID,COVID/images/COVID-1715.png,COVID/masks/COVID-1715.png,True,G00777,train,17798803032da215d58bc06d1c5f6fe5,817b445e1e7a5a78
COVID-1716.png,COVID,COVID/images/COVID-1716.png,COVID/masks/COVID-1716.png,True,G00778,train,5d7a6bc179ced1cce20ed93c4f1a1eeb,8d0d07471e1d6d5d
COVID-1717.png,COVID,COVID/images/COVID-1717.png,COVID/masks/COVID-1717.png,True,G00779,train,92a199ff08b2317e9960b83dbc693267,9b7f0c3b7fe02084
COVID-1718.png,COVID,COVID/images/COVID-1718.png,COVID/masks/COVID-1718.png,True,G00780,train,4cfb97e587a89460256173e219ffd10a,817faed6ea08dc09
COVID-1719.png,COVID,COVID/images/COVID-1719.png,COVID/masks/COVID-1719.png,True,G00781,val,7a197e2a613ad49111af39c8323b9362,d43dc34d5e4b0c5a
COVID-172.png,COVID,COVID/images/COVID-172.png,COVID/masks/COVID-172.png,True,G00782,test,fc9bf801732b4c95e5910ad2d8af9d56,d83dc71e96431716
COVID-1720.png,COVID,COVID/images/COVID-1720.png,COVID/masks/COVID-1720.png,True,G00783,val,dda1c26f7599be49d80c68f45818503d,887fe65e5945581c
COVID-1721.png,COVID,COVID/images/COVID-1721.png,COVID/masks/COVID-1721.png,True,G00784,train,b98cb257c3b7174abec3bdffc750c354,dcad830fd70f0c45
COVID-1722.png,COVID,COVID/images/COVID-1722.png,COVID/masks/COVID-1722.png,True,G00785,train,0efacea780ef9150f36e2f773df1ab9d,9d6ee282de55d540
COVID-1723.png,COVID,COVID/images/COVID-1723.png,COVID/masks/COVID-1723.png,True,G00786,test,07dbdf9514c252ca4652234f80c65db5,9179465a154f5e17
COVID-1724.png,COVID,COVID/images/COVID-1724.png,COVID/masks/COVID-1724.png,True,G00787,train,9898c3efb56de2262d16577d5c628195,cd2cc20fc617dc4d
COVID-1725.png,COVID,COVID/images/COVID-1725.png,COVID/masks/COVID-1725.png,True,G00788,train,8956fd0ab5c72a340df3103b8f83d39c,9128171a565e5bde
COVID-1726.png,COVID,COVID/images/COVID-1726.png,COVID/masks/COVID-1726.png,True,G00789,test,4833744580afc09390bcf0533eca9e2f,dd2cd3531f1d5809
COVID-1727.png,COVID,COVID/images/COVID-1727.png,COVID/masks/COVID-1727.png,True,G00790,test,8b2c7d31ed90c6a1953e0c4388e40a64,c42f431d4c467e4f
COVID-1728.png,COVID,COVID/images/COVID-1728.png,COVID/masks/COVID-1728.png,True,G00791,train,4058151def8898dbb3e33d46b36b3e0d,dc0a831b1d457be6
COVID-1729.png,COVID,COVID/images/COVID-1729.png,COVID/masks/COVID-1729.png,True,G00792,train,85483548e24efa9405cd2fed476b3724,d81b5b0da4bcb5a4
COVID-173.png,COVID,COVID/images/COVID-173.png,COVID/masks/COVID-173.png,True,G00793,train,ba5dd9a06ddeec3ba77b695fc71bf81b,907d47571e1a4f38
COVID-1730.png,COVID,COVID/images/COVID-1730.png,COVID/masks/COVID-1730.png,True,G00794,train,c7dbea060fe62dcc46e0bf913ab9582a,991b463a19433eee
COVID-1731.png,COVID,COVID/images/COVID-1731.png,COVID/masks/COVID-1731.png,True,G00795,train,3e069c3ded19b86bc134796accbbf306,dc2e831b0f243e57
COVID-1732.png,COVID,COVID/images/COVID-1732.png,COVID/masks/COVID-1732.png,True,G00796,train,08d1fa3f89554b354d2b9df615830033,8c0e475b1d4c68fd
COVID-1733.png,COVID,COVID/images/COVID-1733.png,COVID/masks/COVID-1733.png,True,G00797,train,aa4635fc36cc3557664557d211f98fa5,d53b031a494f5d5c
COVID-1734.png,COVID,COVID/images/COVID-1734.png,COVID/masks/COVID-1734.png,True,G00798,train,f146d69faa041c9023999d1a5f97c81f,cd46431f4e465e69
COVID-1735.png,COVID,COVID/images/COVID-1735.png,COVID/masks/COVID-1735.png,True,G00799,train,3bbb4fac9c3404dcdb9f62d75fa44589,c43e53131f45c35d
COVID-1736.png,COVID,COVID/images/COVID-1736.png,COVID/masks/COVID-1736.png,True,G00800,train,dbb176629f1b5bdfb1d3d3bbade10283,843e03531b5f4f6c
COVID-1737.png,COVID,COVID/images/COVID-1737.png,COVID/masks/COVID-1737.png,True,G00801,val,e2ae74b6daf422c84389875caaec8486,c13947461e585edb
COVID-1738.png,COVID,COVID/images/COVID-1738.png,COVID/masks/COVID-1738.png,True,G00802,val,8e0f7c3b38526381f59377d2dea43cea,894e4e184de776c6
COVID-1739.png,COVID,COVID/images/COVID-1739.png,COVID/masks/COVID-1739.png,True,G00803,val,5d5f11754c907b24ab4e25f82613e881,d13a565a58495b5b
COVID-174.png,COVID,COVID/images/COVID-174.png,COVID/masks/COVID-174.png,True,G00804,test,e7ced2dd33cab7aad29dea0a279b5eea,817a161e597939e9
COVID-1740.png,COVID,COVID/images/COVID-1740.png,COVID/masks/COVID-1740.png,True,G00805,train,f0a4ce699f689a6d4dad59f316fa5947,9c0cc61c58cde17f
COVID-1741.png,COVID,COVID/images/COVID-1741.png,COVID/masks/COVID-1741.png,True,G00806,train,ffb210f91f5d543b808294b2e4c10173,947c87511d475d1d
COVID-1742.png,COVID,COVID/images/COVID-1742.png,COVID/masks/COVID-1742.png,True,G00807,train,24fe1fca23dd976b9b166294e8967589,d918871b194df333
COVID-1743.png,COVID,COVID/images/COVID-1743.png,COVID/masks/COVID-1743.png,True,G00808,train,82fcc7d0eaffebaa9ae710e2a5400a28,d43d43431d4e4f5c
COVID-1744.png,COVID,COVID/images/COVID-1744.png,COVID/masks/COVID-1744.png,True,G00809,train,57c0a22782b8c0d0626d53783bf78fb5,dd0ec60b1f4d7039
COVID-1745.png,COVID,COVID/images/COVID-1745.png,COVID/masks/COVID-1745.png,True,G00810,train,62dfc808829b8f8159a8514df07ab760,d43dd35b0f430e4a
COVID-1746.png,COVID,COVID/images/COVID-1746.png,COVID/masks/COVID-1746.png,True,G00811,test,feb338d408721c95654ade6b9331fed7,9819d55f19791137
COVID-1747.png,COVID,COVID/images/COVID-1747.png,COVID/masks/COVID-1747.png,True,G00812,train,818e58e7e635b204929482c20b3f9971,c46c171e190d7e7c
COVID-1748.png,COVID,COVID/images/COVID-1748.png,COVID/masks/COVID-1748.png,True,G00813,train,56a2b404cf3196fb88f6a80f451cd52a,dd1c07071d5c4d6d
COVID-1749.png,COVID,COVID/images/COVID-1749.png,COVID/masks/COVID-1749.png,True,G00814,train,b1d0be833cce6f347a64f7c08d0a924a,81681f1e5a5b78d9
COVID-175.png,COVID,COVID/images/COVID-175.png,COVID/masks/COVID-175.png,True,G00815,train,ee96eed714ccb5b8e3a7f39d2b76407e,b92dc61293c71b1e
COVID-1750.png,COVID,COVID/images/COVID-1750.png,COVID/masks/COVID-1750.png,True,G00816,train,df2548c8569541417356658772fc3a57,d62e8d558e59834b
COVID-1751.png,COVID,COVID/images/COVID-1751.png,COVID/masks/COVID-1751.png,True,G00817,train,3a96f9c90338ad58d3a21d6af98cb21f,95692c141a1f7f78
COVID-1752.png,COVID,COVID/images/COVID-1752.png,COVID/masks/COVID-1752.png,True,G00818,train,e3a8fc915e86708379c5ec9bdce40e8c,dc2ca393ab1d94d4
COVID-1753.png,COVID,COVID/images/COVID-1753.png,COVID/masks/COVID-1753.png,True,G00819,val,eb1d4a8afbc8dcc872536837c032f63c,c12d5a565a5a4a5f
COVID-1754.png,COVID,COVID/images/COVID-1754.png,COVID/masks/COVID-1754.png,True,G00820,val,58796026b8d80ece25c094bc4e656a94,d93bd61a1a1d1b13
COVID-1755.png,COVID,COVID/images/COVID-1755.png,COVID/masks/COVID-1755.png,True,G00821,test,3b53c9de066be55f6b40c436a21a2f8f,c56d16821b175e5b
COVID-1756.png,COVID,COVID/images/COVID-1756.png,COVID/masks/COVID-1756.png,True,G00822,train,5683227dc42dbdb9b9e07eeee2a18a56,8d2ce6baba85c703
COVID-1757.png,COVID,COVID/images/COVID-1757.png,COVID/masks/COVID-1757.png,True,G00823,val,1de049039272ffd58be1a403a46a725c,813956587bcf6991
COVID-1758.png,COVID,COVID/images/COVID-1758.png,COVID/masks/COVID-1758.png,True,G00824,train,e8c1b760a806ef2cee36b37d2c8c9156,dd3ce0210fe78707
COVID-1759.png,COVID,COVID/images/COVID-1759.png,COVID/masks/COVID-1759.png,True,G00825,test,1ffa887da6d43fcd8b2c3b3d772887df,dd28d25b5f44683c
COVID-176.png,COVID,COVID/images/COVID-176.png,COVID/masks/COVID-176.png,True,G00826,train,701fa3e5a51970d87271e029526d0d56,d42f414f0e4e0f6b
COVID-1760.png,COVID,COVID/images/COVID-1760.png,COVID/masks/COVID-1760.png,True,G00827,train,17aa49e4672bca15226623e669e9d16f,c15f4656161b5f0b
COVID-1761.png,COVID,COVID/images/COVID-1761.png,COVID/masks/COVID-1761.png,True,G00828,test,a9a47d01b7c6b4811c39d61765ebf3ce,816c87961b9b3f85
COVID-1762.png,COVID,COVID/images/COVID-1762.png,COVID/masks/COVID-1762.png,True,G00829,train,d2dcc5db752d17bfd48914a6b3694d02,cf76e205a7368178
COVID-1763.png,COVID,COVID/images/COVID-1763.png,COVID/masks/COVID-1763.png,True,G00830,train,1da1909a7ac7da4c905c10a021b73496,dc2e831f0d541e4f
COVID-1764.png,COVID,COVID/images/COVID-1764.png,COVID/masks/COVID-1764.png,True,G00831,train,8a74eb60bf0d1291eb178c0b396a671b,984fc796ce431a5a
COVID-1765.png,COVID,COVID/images/COVID-1765.png,COVID/masks/COVID-1765.png,True,G00832,train,14c86437d7228722260dab8bea2ce2a2,9c6cc7171e421f1b
COVID-1766.png,COVID,COVID/images/COVID-1766.png,COVID/masks/COVID-1766.png,True,G00833,train,00ea43dae61d135330f0ca8c9254169d,9d2de6ee8e084d0d
COVID-1767.png,COVID,COVID/images/COVID-1767.png,COVID/masks/COVID-1767.png,True,G00834,train,ad6e7be4e3613f8b99adb3abac4b867a,c03d135a5b1b0b7e
COVID-1768.png,COVID,COVID/images/COVID-1768.png,COVID/masks/COVID-1768.png,True,G00835,train,bcab43a387a40795d52c6bd402475ee4,913bd786fe188616
COVID-1769.png,COVID,COVID/images/COVID-1769.png,COVID/masks/COVID-1769.png,True,G00836,train,21edd226a21370104fd877a76e34328b,893a16def8e5e0a1
COVID-177.png,COVID,COVID/images/COVID-177.png,COVID/masks/COVID-177.png,True,G00184,train,b580780f52a370eb2ff5514616093b1d,9979c7961e464978
COVID-1770.png,COVID,COVID/images/COVID-1770.png,COVID/masks/COVID-1770.png,True,G00837,train,1fee193dc47317a3d66859ee18b85a0c,c979f6f6ba504818
COVID-1771.png,COVID,COVID/images/COVID-1771.png,COVID/masks/COVID-1771.png,True,G00838,train,364969aaa42f0195eb41e66146ec9f13,c47d47535a1f161a
COVID-1772.png,COVID,COVID/images/COVID-1772.png,COVID/masks/COVID-1772.png,True,G00839,train,bc82d656ed3ece91d10ab05ddfe11217,d939e6c65e9e8809
COVID-1773.png,COVID,COVID/images/COVID-1773.png,COVID/masks/COVID-1773.png,True,G00840,val,4e3c75fc928387ac0e241951c184873b,896805875a9f06ff
COVID-1774.png,COVID,COVID/images/COVID-1774.png,COVID/masks/COVID-1774.png,True,G00841,val,fe876f54030ba3aaf0d2c39518a9d143,d92987eebac2ea01
COVID-1775.png,COVID,COVID/images/COVID-1775.png,COVID/masks/COVID-1775.png,True,G00842,train,830a0b6c7c7f9020acce7debe3580331,95680c8696d2ded7
COVID-1776.png,COVID,COVID/images/COVID-1776.png,COVID/masks/COVID-1776.png,True,G00843,val,82c7dec04ecec09dc301940234809c2c,915f46165e591f0b
COVID-1777.png,COVID,COVID/images/COVID-1777.png,COVID/masks/COVID-1777.png,True,G00844,test,0a8607acd693a46908ea65c9e4b3d8b8,d13ad65618571b1e
COVID-1778.png,COVID,COVID/images/COVID-1778.png,COVID/masks/COVID-1778.png,True,G00845,test,3ea54492c673012ec8d79e51f66943f5,d52f47be2941051f
COVID-1779.png,COVID,COVID/images/COVID-1779.png,COVID/masks/COVID-1779.png,True,G00846,train,43cbae3f8fcb05135b530ea5ed7330cf,8360051d17175f7f
COVID-178.png,COVID,COVID/images/COVID-178.png,COVID/masks/COVID-178.png,True,G00847,train,c7d18c7f5514fad074d936a43b74a913,9d6cc01b4f4d1a5d
COVID-1780.png,COVID,COVID/images/COVID-1780.png,COVID/masks/COVID-1780.png,True,G00848,train,92497104c9f9ee9ee877ce39c33b5ca0,cd3ce6923b3cbc10
COVID-1781.png,COVID,COVID/images/COVID-1781.png,COVID/masks/COVID-1781.png,True,G00849,train,5f36189a0dfb330ddfb9b73f12de58c8,81797fc7869292c5
COVID-1782.png,COVID,COVID/images/COVID-1782.png,COVID/masks/COVID-1782.png,True,G00850,train,316c3f9b2c706e9df7ec2b649c7d4aa9,9c28e3e7bb1b4909
COVID-1783.png,COVID,COVID/images/COVID-1783.png,COVID/masks/COVID-1783.png,True,G00851,train,5a50a62cf6cc894677c0716c86163c1e,d42e42574f4c6e4e
COVID-1784.png,COVID,COVID/images/COVID-1784.png,COVID/masks/COVID-1784.png,True,G00852,train,05e2611079547972200c4b473d5b8c8c,f961c71ecc144e4d
COVID-1785.png,COVID,COVID/images/COVID-1785.png,COVID/masks/COVID-1785.png,True,G00853,train,d83b5c4b739ab515d1008fb57b077656,c17a544c79696f23
COVID-1786.png,COVID,COVID/images/COVID-1786.png,COVID/masks/COVID-1786.png,True,G00854,train,9a4f0bd661744d9bfda926bdaf400a0c,f929e7165cc4921e
COVID-1787.png,COVID,COVID/images/COVID-1787.png,COVID/masks/COVID-1787.png,True,G00855,train,eeb81ba7c4f4d2b803f0b60fd3d9b740,c03d5b035f5f284e
COVID-1788.png,COVID,COVID/images/COVID-1788.png,COVID/masks/COVID-1788.png,True,G00856,train,9c872aed5714fa08cf70d143d0c599f4,d93ba3da5958521a
COVID-1789.png,COVID,COVID/images/COVID-1789.png,COVID/masks/COVID-1789.png,True,G00857,train,f4fe2f04fb2afbb1ca42f8a1ca4c8a78,817b06863a1dce5f
COVID-179.png,COVID,COVID/images/COVID-179.png,COVID/masks/COVID-179.png,True,G00858,train,b0fc7a7dc66ff0e22a6f163012726364,946f831b0d4f3c6a
COVID-1790.png,COVID,COVID/images/COVID-1790.png,COVID/masks/COVID-1790.png,True,G00859,train,d108ed92e7a38f8a5012b05755164d56,d92cc592c547b547
COVID-1791.png,COVID,COVID/images/COVID-1791.png,COVID/masks/COVID-1791.png,True,G00860,train,5b94711427154172181faadf921b7be0,913fc69279455959
COVID-1792.png,COVID,COVID/images/COVID-1792.png,COVID/masks/COVID-1792.png,True,G00861,val,ee13d18361387f88069119b88416869d,dd2e52434f461e0f
COVID-1793.png,COVID,COVID/images/COVID-1793.png,COVID/masks/COVID-1793.png,True,G00862,test,c0b2b7142b2356a0be6db4f0d58a47cd,d62fc1d105462f1f
COVID-1794.png,COVID,COVID/images/COVID-1794.png,COVID/masks/COVID-1794.png,True,G00863,val,4700a39e25585c7dd6cb49921459875a,9869051e4a5f7a79
COVID-1795.png,COVID,COVID/images/COVID-1795.png,COVID/masks/COVID-1795.png,True,G00864,train,4e196a7a9f2aaa76274eb019ece61b5c,dd2c07d319594736
COVID-1796.png,COVID,COVID/images/COVID-1796.png,COVID/masks/COVID-1796.png,True,G00865,val,5b92eac17fda0e3d7aa9da4833172b0a,d969e7965e411859
COVID-1797.png,COVID,COVID/images/COVID-1797.png,COVID/masks/COVID-1797.png,True,G00866,test,133a92d25c9480a3761b1ce21c1b39d5,d62b092f3e3bd1c0
COVID-1798.png,COVID,COVID/images/COVID-1798.png,COVID/masks/COVID-1798.png,True,G00867,test,63957d035a4b5e7a8121aeec4f4eda73,d83fd65a1c4e191c
COVID-1799.png,COVID,COVID/images/COVID-1799.png,COVID/masks/COVID-1799.png,True,G00868,train,3f2831c748be9bc9682b278780944404,d515954a3a6a6af1
COVID-18.png,COVID,COVID/images/COVID-18.png,COVID/masks/COVID-18.png,True,G00869,train,a00f1e48eaebd032eee090def7795dac,816c0e1a5f7d7c4c
COVID-180.png,COVID,COVID/images/COVID-180.png,COVID/masks/COVID-180.png,True,G00186,train,d9770d962a24897f6013f01866b16761,896fc65b4d1139a6
COVID-1800.png,COVID,COVID/images/COVID-1800.png,COVID/masks/COVID-1800.png,True,G00870,train,f986bc2bfde4c8469c96e190142ca2ba,8d7cc25b1c691c5e
COVID-1801.png,COVID,COVID/images/COVID-1801.png,COVID/masks/COVID-1801.png,True,G00871,train,96d02325e9746ce316be199af69985c9,d170d4195f571555
COVID-1802.png,COVID,COVID/images/COVID-1802.png,COVID/masks/COVID-1802.png,True,G00872,train,68f1d64916baf6f0a1105417d625adf2,cc3fb0601b65d173
COVID-1803.png,COVID,COVID/images/COVID-1803.png,COVID/masks/COVID-1803.png,True,G00873,train,1460224fdc62cdd2e1c6f841d49b2222,dc0bc34ed253c2ea
COVID-1804.png,COVID,COVID/images/COVID-1804.png,COVID/masks/COVID-1804.png,True,G00874,train,ba34d8123e2690d8e299b1a2c8495d61,990d471e1a791c5f
COVID-1805.png,COVID,COVID/images/COVID-1805.png,COVID/masks/COVID-1805.png,True,G00875,train,b752ee57c76deae0bd196c48e9700983,942e07571f0e4757
COVID-1806.png,COVID,COVID/images/COVID-1806.png,COVID/masks/COVID-1806.png,True,G00876,train,efe8dc8ab2697687e10d916796a545e1,953ac2171e551f1e
COVID-1807.png,COVID,COVID/images/COVID-1807.png,COVID/masks/COVID-1807.png,True,G00877,train,73918eedb0e13acc8799183bb31a1c54,d43d02535f1e4357
COVID-1808.png,COVID,COVID/images/COVID-1808.png,COVID/masks/COVID-1808.png,True,G00878,train,6e6c343c3e0c067cb13615b7aa839b00,8378041c795b7a7b
COVID-1809.png,COVID,COVID/images/COVID-1809.png,COVID/masks/COVID-1809.png,True,G00879,val,2c49be3f04c5f67f466a2f1e8bcaafb8,cc2e5243a94adf4b
COVID-181.png,COVID,COVID/images/COVID-181.png,COVID/masks/COVID-181.png,True,G00013,train,452e5741b587b2c648d2dfa47a9fe728,997b87161e491d1b
COVID-1810.png,COVID,COVID/images/COVID-1810.png,COVID/masks/COVID-1810.png,True,G00880,val,522c4798971f4115c1a323914d0760b5,d52ac71a1d4d791c
COVID-1811.png,COVID,COVID/images/COVID-1811.png,COVID/masks/COVID-1811.png,True,G00881,test,1809ba2cfb0688c142ba7529d3fde908,d87bc41ec7161e4a
COVID-1812.png,COVID,COVID/images/COVID-1812.png,COVID/masks/COVID-1812.png,True,G00882,val,9f1292f526cd4b65ea4572461940f101,897986161a0f3f3b
COVID-1813.png,COVID,COVID/images/COVID-1813.png,COVID/masks/COVID-1813.png,True,G00883,test,efb3f95fca78b35f36dad8c9ba67edb7,9c69c31e4a3d705b
COVID-1814.png,COVID,COVID/images/COVID-1814.png,COVID/masks/COVID-1814.png,True,G00884,train,ea4c6326e4d2af4210d67ad03d76a4b7,d07fc6d2ebc0e0c8
COVID-1815.png,COVID,COVID/images/COVID-1815.png,COVID/masks/COVID-1815.png,True,G00885,test,2cbb5aa2ed2576735ae8f15aa7487c26,9969c61ec71e1d0d
COVID-1816.png,COVID,COVID/images/COVID-1816.png,COVID/masks/COVID-1816.png,True,G00886,train,e166f3b311f044070e9596035c47a907,d91f8e0987a3a5e1
COVID-1817.png,COVID,COVID/images/COVID-1817.png,COVID/masks/COVID-1817.png,True,G00887,train,0d9c2549f6f5482bfd0f33bf5e694cce,986dc692c31f1e4b
COVID-1818.png,COVID,COVID/images/COVID-1818.png,COVID/masks/COVID-1818.png,True,G00888,train,2cae0bdad731c00acd15096c25e20805,9929c61a5d5b194f
COVID-1819.png,COVID,COVID/images/COVID-1819.png,COVID/masks/COVID-1819.png,True,G00889,train,f3fcdecdd0cf569394a395838269ae65,b96d471e4d1a1c4b
COVID-182.png,COVID,COVID/images/COVID-182.png,COVID/masks/COVID-182.png,True,G00890,train,a19da82a99a2897bc29d46c24eb7811b,d17bc6425e455751
COVID-1820.png,COVID,COVID/images/COVID-1820.png,COVID/masks/COVID-1820.png,True,G00891,train,1ac8c28447430ebbc3bc6f5324709650,d42f415e2e2c6c2f
COVID-1821.png,COVID,COVID/images/COVID-1821.png,COVID/masks/COVID-1821.png,True,G00892,train,908094a4cdf60b72abc82682cd51cd67,dd2c530b0d2c3cbd
COVID-1822.png,COVID,COVID/images/COVID-1822.png,COVID/masks/COVID-1822.png,True,G00893,train,117c9e761e1cedfce21a8bd0eb088a9f,912a06194e78fdfc
COVID-1823.png,COVID,COVID/images/COVID-1823.png,COVID/masks/COVID-1823.png,True,G00894,train,b3f97f9a7db618aa6902c7f2f9b6f805,8c2e40533b797c7c
COVID-1824.png,COVID,COVID/images/COVID-1824.png,COVID/masks/COVID-1824.png,True,G00895,train,4367658d0d05b21fa4582910bceb7e8f,cd6e52434b4d4d4e
COVID-1825.png,COVID,COVID/images/COVID-1825.png,COVID/masks/COVID-1825.png,True,G00896,train,83ae784b23a96b8e70cf084e8d4c8d9c,9c2f035b0f4c5e4d
COVID-1826.png,COVID,COVID/images/COVID-1826.png,COVID/masks/COVID-1826.png,True,G00897,train,7ed05a5f12a7584d3d1de489b8a7fae3,d42e26d3d35991c3
COVID-1827.png,COVID,COVID/images/COVID-1827.png,COVID/masks/COVID-1827.png,True,G00898,val,642b090391898ace27ec313aca10e7df,986dc616c1571f17
COVID-1828.png,COVID,COVID/images/COVID-1828.png,COVID/masks/COVID-1828.png,True,G00899,train,36cb6a596968d2c0e78f6e784da835f1,d42f42d34b4d4e4e
COVID-1829.png,COVID,COVID/images/COVID-1829.png,COVID/masks/COVID-1829.png,True,G00900,train,81d983fc550dce74e45c2b515e11bed2,bd2cc05257c11bcf
COVID-183.png,COVID,COVID/images/COVID-183.png,COVID/masks/COVID-183.png,True,G00901,train,d96bba153d71d4f996711f1603b1fe39,d96c461e1b0f1d4e
COVID-1830.png,COVID,COVID/images/COVID-1830.png,COVID/masks/COVID-1830.png,True,G00902,test,15cdbf1fba5e74dcbfdda70668700c8e,903f565a4c5b1d4e
COVID-1831.png,COVID,COVID/images/COVID-1831.png,COVID/masks/COVID-1831.png,True,G00903,test,5ab7c0180ec5e270c69ef3a531e6828a,d92dc21acf58bd48
COVID-1832.png,COVID,COVID/images/COVID-1832.png,COVID/masks/COVID-1832.png,True,G00904,test,25530f4c1710215b655c705de51bbcbe,d42ed35d434b1d54
COVID-1833.png,COVID,COVID/images/COVID-1833.png,COVID/masks/COVID-1833.png,True,G00905,train,c3a093a4dc1d92a24a5241916a89bc6e,d93dd0531742195f
COVID-1834.png,COVID,COVID/images/COVID-1834.png,COVID/masks/COVID-1834.png,True,G00906,train,72a7b73ee25b692a5ab188a94439cc0c,d92bd25b54435d58
COVID-1835.png,COVID,COVID/images/COVID-1835.png,COVID/masks/COVID-1835.png,True,G00907,val,8a56307e4e5bb0afbda66ed240b0792f,d82d87161b4f1e5a
COVID-1836.png,COVID,COVID/images/COVID-1836.png,COVID/masks/COVID-1836.png,True,G00908,train,cf0c8c24e2b938f65dfcc0b4de45515f,dd2e53435f0d4c58
COVID-1837.png,COVID,COVID/images/COVID-1837.png,COVID/masks/COVID-1837.png,True,G00909,test,97dbe1a453abb2eb69815618fb4b5646,d8795356164a5b1b
COVID-1838.png,COVID,COVID/images/COVID-1838.png,COVID/masks/COVID-1838.png,True,G00910,val,a360b1c82a713fcac3e6243faf55690d,d42c53c34f4f6e44
COVID-1839.png,COVID,COVID/images/COVID-1839.png,COVID/masks/COVID-1839.png,True,G00911,train,97d9f47f73d7e882cc8e930126edf2ca,dc39c20fcf1e1a58
COVID-184.png,COVID,COVID/images/COVID-184.png,COVID/masks/COVID-184.png,True,G00912,train,b97c9dacf8908dd4a5b7114c7d30f156,cc3d51a356d556c2
COVID-1840.png,COVID,COVID/images/COVID-1840.png,COVID/masks/COVID-1840.png,True,G00913,train,08a0b8417c94e89a7b51196b40efba34,a87d851e5a491d5e
COVID-1841.png,COVID,COVID/images/COVID-1841.png,COVID/masks/COVID-1841.png,True,G00914,train,3819ab370fd240e15985b0c691ba15e2,8d7cd00b4d0d5f69
COVID-1842.png,COVID,COVID/images/COVID-1842.png,COVID/masks/COVID-1842.png,True,G00915,train,9ad9a92d98a26fcbd191921e297e2cbd,83a17c9c5c5cdc78
COVID-1843.png,COVID,COVID/images/COVID-1843.png,COVID/masks/COVID-1843.png,True,G00916,train,b55fc54ad8bcca494fbddfa577363590,d8f3856a271a4e1e
COVID-1844.png,COVID,COVID/images/COVID-1844.png,COVID/masks/COVID-1844.png,True,G00917,train,eeb8c764098d49f1b938710a528a3bb4,91e886171b879f1b
COVID-1845.png,COVID,COVID/images/COVID-1845.png,COVID/masks/COVID-1845.png,True,G00918,train,603665ce99bd267189893185a27caa8d,d87961c596de4287
COVID-1846.png,COVID,COVID/images/COVID-1846.png,COVID/masks/COVID-1846.png,True,G00919,train,a75176162aa408cff7aa04bd7bc383ef,c1388646dbdb8657
COVID-1847.png,COVID,COVID/images/COVID-1847.png,COVID/masks/COVID-1847.png,True,G00920,train,0d9e61bc8e181057110014e5c4eb1010,9939e587964b43f0
COVID-1848.png,COVID,COVID/images/COVID-1848.png,COVID/masks/COVID-1848.png,True,G00921,val,56cc22e553fc7deb21c96e807cbaae2c,897d86461b17875b
COVID-1849.png,COVID,COVID/images/COVID-1849.png,COVID/masks/COVID-1849.png,True,G00922,test,387cfff46916c683afb4c4ec6a9746ef,d97a966b79323111
COVID-185.png,COVID,COVID/images/COVID-185.png,COVID/masks/COVID-185.png,True,G00923,test,5979765fb69d40315f4591f415b08096,915f96186973ea61
COVID-1850.png,COVID,COVID/images/COVID-1850.png,COVID/masks/COVID-1850.png,True,G00924,val,9e2d746479c68e8f2b8b1c12f127de28,d46c66931bc52bd1
COVID-1851.png,COVID,COVID/images/COVID-1851.png,COVID/masks/COVID-1851.png,True,G00925,train,9929840bc7ac9c7ce1b0bfa22a8f27e2,9929c51696371e4f
COVID-1852.png,COVID,COVID/images/COVID-1852.png,COVID/masks/COVID-1852.png,True,G00926,train,b2c3d552db8d0cc00cfd99e8cd630557,f965865a0d5f505c
COVID-1853.png,COVID,COVID/images/COVID-1853.png,COVID/masks/COVID-1853.png,True,G00927,train,eab431a6bb0a4e96ce44aa5bf576dddf,f979159616070f0f
COVID-1854.png,COVID,COVID/images/COVID-1854.png,COVID/masks/COVID-1854.png,True,G00928,train,a868c36bf955c5d9c85ac62704f27faa,c12e825b0d557c7e
COVID-1855.png,COVID,COVID/images/COVID-1855.png,COVID/masks/COVID-1855.png,True,G00929,train,32be51b6583870a9de1eb96418abe36d,b81945161e5b5d5e
COVID-1856.png,COVID,COVID/images/COVID-1856.png,COVID/masks/COVID-1856.png,True,G00930,val,d0a02b7f97587626c2109a7fa86dc20e,d52f42571f464b1c
COVID-1857.png,COVID,COVID/images/COVID-1857.png,COVID/masks/COVID-1857.png,True,G00931,test,c1ec68048e9c38c99e8be70d6a2fa6c4,c52e565b484d4e4f
COVID-1858.png,COVID,COVID/images/COVID-1858.png,COVID/masks/COVID-1858.png,True,G00932,val,706500e606d18e345186145f72de2181,817b07061f554f1f
COVID-1859.png,COVID,COVID/images/COVID-1859.png,COVID/masks/COVID-1859.png,True,G00933,train,d6963fc8208a1342413f5ed6bd746bd8,c42f530b0c2eb9bb
COVID-186.png,COVID,COVID/images/COVID-186.png,COVID/masks/COVID-186.png,True,G00213,train,4288dae5c9eb9abb5ff33668ea206bd3,9979861e1c5b1b59
COVID-1860.png,COVID,COVID/images/COVID-1860.png,COVID/masks/COVID-1860.png,True,G00934,train,1b0839e4c94d06e401a7b95a53b2e375,d43dc34d4f43490f
COVID-1861.png,COVID,COVID/images/COVID-1861.png,COVID/masks/COVID-1861.png,True,G00935,train,4799e19b196d2f2c2dc709effe4367cc,817c7a461a5a5d6d
COVID-1862.png,COVID,COVID/images/COVID-1862.png,COVID/masks/COVID-1862.png,True,G00936,val,c4eb3ff6394986148ef71eaea946cc44,9e9e718c9c718e8c
COVID-1863.png,COVID,COVID/images/COVID-1863.png,COVID/masks/COVID-1863.png,True,G00937,train,e4a8a40a78e493f8c2812f974256fa85,816fc65e0d0b0b7d
COVID-1864.png,COVID,COVID/images/COVID-1864.png,COVID/masks/COVID-1864.png,True,G00938,train,9691f1c3c1efa71758f6abeb4cc398ae,817b7e967ac06e81
COVID-1865.png,COVID,COVID/images/COVID-1865.png,COVID/masks/COVID-1865.png,True,G00939,train,a78820f600a8d749afa141ee71e82a5c,9c25821f4a255efd
COVID-1866.png,COVID,COVID/images/COVID-1866.png,COVID/masks/COVID-1866.png,True,G00940,train,60ee741aca1a31bdfe583aae4eb3d2b1,c46e373d3864e23c
COVID-1867.png,COVID,COVID/images/COVID-1867.png,COVID/masks/COVID-1867.png,True,G00941,train,a5b0e4cc5ee061ea8ceb924f6eec90ed,d82d965a5fd21a8c
COVID-1868.png,COVID,COVID/images/COVID-1868.png,COVID/masks/COVID-1868.png,True,G00942,val,129cdf80690873e7f0ab974e4e1dddbe,c46e6f5a7868e0c9
COVID-1869.png,COVID,COVID/images/COVID-1869.png,COVID/masks/COVID-1869.png,True,G00943,train,b31186c65e867504eef41f8a57e04216,887fe75e1841531b
COVID-187.png,COVID,COVID/images/COVID-187.png,COVID/masks/COVID-187.png,True,G00944,train,4d439614b18763d1059e9e99dbbda79a,d42f23d12bc02f9d
COVID-1870.png,COVID,COVID/images/COVID-1870.png,COVID/masks/COVID-1870.png,True,G00945,test,958143ee32ac4cec866b4e7680430231,d46e03971919ff42
COVID-1871.png,COVID,COVID/images/COVID-1871.png,COVID/masks/COVID-1871.png,True,G00946,test,bc8aac7de39691fe53a05a6bb5afbf7c,cd2d06531f4d4a1f
COVID-1872.png,COVID,COVID/images/COVID-1872.png,COVID/masks/COVID-1872.png,True,G00947,train,9cf1d90495d693fc011dd0e54f4e8f53,c46eb74e6ce289c1
COVID-1873.png,COVID,COVID/images/COVID-1873.png,COVID/masks/COVID-1873.png,True,G00948,train,ee5ffd4e9153c5d2a4108f95087d51ef,dd2d421b1d4d5c6a
COVID-1874.png,COVID,COVID/images/COVID-1874.png,COVID/masks/COVID-1874.png,True,G00949,train,287c7b90590e8ba31d1785b50ad60258,8938161979e774e5
COVID-1875.png,COVID,COVID/images/COVID-1875.png,COVID/masks/COVID-1875.png,True,G00950,test,dcb460d642d1f780076b79bc564f3e28,9979461e5d5a1e58
COVID-1876.png,COVID,COVID/images/COVID-1876.png,COVID/masks/COVID-1876.png,True,G00951,train,9de0cb1c021fe0d4fe4e4ba5ef87caa6,c43d17c228ef6f90
COVID-1877.png,COVID,COVID/images/COVID-1877.png,COVID/masks/COVID-1877.png,True,G00952,train,961743c6ceee6af3f48de984c7273df4,dc2d065a491e3d5d
COVID-1878.png,COVID,COVID/images/COVID-1878.png,COVID/masks/COVID-1878.png,True,G00953,test,6386f5740115429ba496c4fa44240119,993b87161a1d5b1b
COVID-1879.png,COVID,COVID/images/COVID-1879.png,COVID/masks/COVID-1879.png,True,G00954,train,36ab3ccf684dc767540df586772918e9,997d451e1e491f1c
COVID-188.png,COVID,COVID/images/COVID-188.png,COVID/masks/COVID-188.png,True,G00955,train,af718fa998dd57c264df50f85ec01517,dc2ae15d4f541a59
COVID-1880.png,COVID,COVID/images/COVID-1880.png,COVID/masks/COVID-1880.png,True,G00956,train,e5af38106698f96e70e958ed4601ad81,8c4f03596ca6f3a3
COVID-1881.png,COVID,COVID/images/COVID-1881.png,COVID/masks/COVID-1881.png,True,G00957,train,26d48323430fb1ee01f72e2a2195f2ea,9d09458747397d78
COVID-1882.png,COVID,COVID/images/COVID-1882.png,COVID/masks/COVID-1882.png,True,G00958,train,a1f0adfdcccd79dbe68b1395736cb2ed,c0b68b4b0f0f0f3e
COVID-1883.png,COVID,COVID/images/COVID-1883.png,COVID/masks/COVID-1883.png,True,G00959,train,862ebca7ea6206f750868522593d8afd,847ed35b4d843e2c
COVID-1884.png,COVID,COVID/images/COVID-1884.png,COVID/masks/COVID-1884.png,True,G00960,train,8c71773ff05aa49b4e316a3740397317,a83e914b1f0b1f0f
COVID-1885.png,COVID,COVID/images/COVID-1885.png,COVID/masks/COVID-1885.png,True,G00961,val,98092700d78c2f35c8c8fc37abbafb40,c8699516fa17790d
COVID-1886.png,COVID,COVID/images/COVID-1886.png,COVID/masks/COVID-1886.png,True,G00962,val,1f47dbb84239c54fca21dac1a6d8cb80,d52cc25b4e4d3b4c
COVID-1887.png,COVID,COVID/images/COVID-1887.png,COVID/masks/COVID-1887.png,True,G00963,train,fd407e215d7a661027a200a40af9d146,817f571a1259787c
COVID-1888.png,COVID,COVID/images/COVID-1888.png,COVID/masks/COVID-1888.png,True,G00964,train,10be219a650ced3741ac566c82f3cb8a,912ec6174c5d1f4d
COVID-1889.png,COVID,COVID/images/COVID-1889.png,COVID/masks/COVID-1889.png,True,G00965,val,1a8d012f15bec92de43dff80fa981ed4,d42ad75f1c421f1c
COVID-189.png,COVID,COVID/images/COVID-189.png,COVID/masks/COVID-189.png,True,G00966,train,68dc3953d4149f0689bd2088089285c4,9869e5163ee570e4
COVID-1890.png,COVID,COVID/images/COVID-1890.png,COVID/masks/COVID-1890.png,True,G00967,test,47cb30b163c7f7cbf9d04bee10b42ed3,997e1618193daca7
COVID-1891.png,COVID,COVID/images/COVID-1891.png,COVID/masks/COVID-1891.png,True,G00968,train,0ac394fffafce3f99723c368274fc854,9031cfc74cc6c55b
COVID-1892.png,COVID,COVID/images/COVID-1892.png,COVID/masks/COVID-1892.png,True,G00969,train,c25db293315c2b8fbc971347d7517aee,d42cd3474d591b0f
COVID-1893.png,COVID,COVID/images/COVID-1893.png,COVID/masks/COVID-1893.png,True,G00970,test,6545ddd160524aac3c4f33fde083c075,b929c5c6165b475c
COVID-1894.png,COVID,COVID/images/COVID-1894.png,COVID/masks/COVID-1894.png,True,G00971,train,0a1bba5afd7cb9a7e8370be5309b81fa,952f861e1c1f595c
COVID-1895.png,COVID,COVID/images/COVID-1895.png,COVID/masks/COVID-1895.png,True,G00972,train,2d45ab5d5f63e765efec48d0d2713164,d62fc9414d57414f
COVID-1896.png,COVID,COVID/images/COVID-1896.png,COVID/masks/COVID-1896.png,True,G00973,train,4a01a4322a6413c98a12d0d8bdd02c7d,c43f534b9b0cecb0
COVID-1897.png,COVID,COVID/images/COVID-1897.png,COVID/masks/COVID-1897.png,True,G00974,train,a3e6873011b79c6b02d717964a9ea902,c53e064b164a4f3f
COVID-1898.png,COVID,COVID/images/COVID-1898.png,COVID/masks/COVID-1898.png,True,G00975,train,0c683a17dea47dbef535a2e0dadbd533,885d47465f0d6f1c
COVID-1899.png,COVID,COVID/images/COVID-1899.png,COVID/masks/COVID-1899.png,True,G00976,val,0a8bdd750dc05b9a15940be26d32b74c,dd2a421f5c4d3b4c
COVID-19.png,COVID,COVID/images/COVID-19.png,COVID/masks/COVID-19.png,True,G00977,train,89da97b327b4eb7afa8bcdbbe48b5112,91696e16181e5f3b
COVID-190.png,COVID,COVID/images/COVID-190.png,COVID/masks/COVID-190.png,True,G00079,train,0693072dfc5ee33da9f07a2c4b32e0d0,c53e27937ac17ac0
COVID-1900.png,COVID,COVID/images/COVID-1900.png,COVID/masks/COVID-1900.png,True,G00978,test,a3dff31c91181a468d6c87d6dea73190,cc35d3510fc3691b
COVID-1901.png,COVID,COVID/images/COVID-1901.png,COVID/masks/COVID-1901.png,True,G00979,train,8be284444dbef09e2ab87ce19016134d,cc35d34b0f631c1e
COVID-1902.png,COVID,COVID/images/COVID-1902.png,COVID/masks/COVID-1902.png,True,G00980,train,530e3ed1e6c1d6c8c315ae357927a8ad,c92e861c1e3b3cea
COVID-1903.png,COVID,COVID/images/COVID-1903.png,COVID/masks/COVID-1903.png,True,G00981,train,a49ab1c616237eb27a529015ab159d8f,997f561879693921
COVID-1904.png,COVID,COVID/images/COVID-1904.png,COVID/masks/COVID-1904.png,True,G00982,val,c2c07ca06ee5272d19ba2daac0ae69d2,d52e434f4d4c0e4f
COVID-1905.png,COVID,COVID/images/COVID-1905.png,COVID/masks/COVID-1905.png,True,G00983,test,2a0f3e9219773a5596ba2a9d533d4b35,913f55c25a5d5549
COVID-1906.png,COVID,COVID/images/COVID-1906.png,COVID/masks/COVID-1906.png,True,G00984,val,7f43ee2eea6b638822d08eb69d21622e,a92ec69b544b195e
COVID-1907.png,COVID,COVID/images/COVID-1907.png,COVID/masks/COVID-1907.png,True,G00985,train,268100ce5f3d59edeb1bd9f3964f7987,d637d0c0079be666
COVID-1908.png,COVID,COVID/images/COVID-1908.png,COVID/masks/COVID-1908.png,True,G00986,train,9e7be7c578636bfcba4899f4db4df056,e96fc669e1e1a085
COVID-1909.png,COVID,COVID/images/COVID-1909.png,COVID/masks/COVID-1909.png,True,G00987,train,38172df30530f1fd352bfd347d3595ae,c52f074f0d4c4f4b
COVID-191.png,COVID,COVID/images/COVID-191.png,COVID/masks/COVID-191.png,True,G00988,train,2e3e1ed37922407f8658d8763d6ec069,d42e530b5a086f5f
COVID-1910.png,COVID,COVID/images/COVID-1910.png,COVID/masks/COVID-1910.png,True,G00989,train,0a5195d6d7e1c5af7c1535dc1e1d0601,d52836939b45d792
COVID-1911.png,COVID,COVID/images/COVID-1911.png,COVID/masks/COVID-1911.png,True,G00990,train,48f66b3e246aa5556170d0826558f0f5,817b14165e5b1e1f
COVID-1912.png,COVID,COVID/images/COVID-1912.png,COVID/masks/COVID-1912.png,True,G00991,train,d3ab0dfcc83ebe18aa3aa4b61c037a3c,853ad7535d44591d
COVID-1913.png,COVID,COVID/images/COVID-1913.png,COVID/masks/COVID-1913.png,True,G00992,train,2ce8d4410cb02a3f67c4c9b6823392cf,c93f12491cda3b9c
COVID-1914.png,COVID,COVID/images/COVID-1914.png,COVID/masks/COVID-1914.png,True,G00993,train,dfa99d2331b4a3bcb7ac2c6066d5eaa0,d47f03438e1c4f1d
COVID-1915.png,COVID,COVID/images/COVID-1915.png,COVID/masks/COVID-1915.png,True,G00994,train,8c9bf5f9153dde9a0002452d9a9cebf5,817986161a5f1f5b
COVID-1916.png,COVID,COVID/images/COVID-1916.png,COVID/masks/COVID-1916.png,True,G00995,train,3b5166efb15e849be10e124bcf30c364,c16b56161eda5b58
COVID-1917.png,COVID,COVID/images/COVID-1917.png,COVID/masks/COVID-1917.png,True,G00996,train,695d03679faf09d851a3878b8a77333c,d03b131e1e7a6c6a
COVID-1918.png,COVID,COVID/images/COVID-1918.png,COVID/masks/COVID-1918.png,True,G00997,train,fb68b0de20024c957d55313ca9aee6dd,981b871e185b5b4f
COVID-1919.png,COVID,COVID/images/COVID-1919.png,COVID/masks/COVID-1919.png,True,G00998,train,b27918eb6f36272facfe98e2e0ee6559,d02bc70a3f85eb38
COVID-192.png,COVID,COVID/images/COVID-192.png,COVID/masks/COVID-192.png,True,G00209,test,dd5a041305ecb568c24667e876d2c85a,d939d6d65a485859
COVID-1920.png,COVID,COVID/images/COVID-1920.png,COVID/masks/COVID-1920.png,True,G00999,val,f2907c21e12794f3acd1447a3cc8065e,887d471e39e9a4b4
COVID-1921.png,COVID,COVID/images/COVID-1921.png,COVID/masks/COVID-1921.png,True,G01000,val,0f9a591731e6c277a2091bd40ec4eeb2,dc2dc3db1d486958
COVID-1922.png,COVID,COVID/images/COVID-1922.png,COVID/masks/COVID-1922.png,True,G01001,val,5aeccc99275f57d2ae092bd5f61c2215,985f161a59591e97
COVID-1923.png,COVID,COVID/images/COVID-1923.png,COVID/masks/COVID-1923.png,True,G01002,test,5b29c545527a324a29838ba6f54f70ce,995d87161b4b1d1e
COVID-1924.png,COVID,COVID/images/COVID-1924.png,COVID/masks/COVID-1924.png,True,G01003,test,ba44ee9f8c0912f6fd823d5c54cf9054,9979071e1e59195d
COVID-1925.png,COVID,COVID/images/COVID-1925.png,COVID/masks/COVID-1925.png,True,G01004,val,efb7fbb0dbd491183ed5bc171a8ece28,d42ad34f5e6c684a
COVID-1926.png,COVID,COVID/images/COVID-1926.png,COVID/masks/COVID-1926.png,True,G01005,test,bba3a7c7da0364b3fa866d4a18fa057f,c90d8547164d5f1f
COVID-1927.png,COVID,COVID/images/COVID-1927.png,COVID/masks/COVID-1927.png,True,G01006,train,9c64e8ec58b4e11757a7a89815919fac,817d061a1d5f1b4f
COVID-1928.png,COVID,COVID/images/COVID-1928.png,COVID/masks/COVID-1928.png,True,G01007,train,f21f95a24d234c72a9cc2d212627cdae,9965965a2d382b3d
COVID-1929.png,COVID,COVID/images/COVID-1929.png,COVID/masks/COVID-1929.png,True,G01008,train,027a493ec5ce229861afc596f59da899,dc7dd21ac5461ae1
COVID-193.png,COVID,COVID/images/COVID-193.png,COVID/masks/COVID-193.png,True,G01009,train,eb764a8a67253ea5152eac69af7b739a,dc1c430b5d4c5d5e
COVID-1930.png,COVID,COVID/images/COVID-1930.png,COVID/masks/COVID-1930.png,True,G01010,train,59bcd945bebfa7e6132d04d1f36a0685,dc2f034d0e3c2f5a
COVID-1931.png,COVID,COVID/images/COVID-1931.png,COVID/masks/COVID-1931.png,True,G01011,train,648c4ea11c2e55503ded8533bc05180b,d430d14ba26f0f6e
COVID-1932.png,COVID,COVID/images/COVID-1932.png,COVID/masks/COVID-1932.png,True,G01012,train,8253bd2c74b78baf6a457eefa56d9973,807fa25a1dc7523b
COVID-1933.png,COVID,COVID/images/COVID-1933.png,COVID/masks/COVID-1933.png,True,G01013,train,2ae4b45af5f39a82e59220e7a140c241,993b461b5c4f194d
COVID-1934.png,COVID,COVID/images/COVID-1934.png,COVID/masks/COVID-1934.png,True,G01014,train,677d7910572b7dac1edec61934146930,9968061a595d797d
COVID-1935.png,COVID,COVID/images/COVID-1935.png,COVID/masks/COVID-1935.png,True,G01015,val,16ff2471d15eb1627ea18f0f5bb1c0c9,85e2bff0449a8b0f
COVID-1936.png,COVID,COVID/images/COVID-1936.png,COVID/masks/COVID-1936.png,True,G01016,train,5bd3339aa85846a909eeefebe3674986,c56c061b7dc539d1
COVID-1937.png,COVID,COVID/images/COVID-1937.png,COVID/masks/COVID-1937.png,True,G01017,train,18b210f7f122af4942f198465fb2e391,917dc61e184f1f1c
COVID-1938.png,COVID,COVID/images/COVID-1938.png,COVID/masks/COVID-1938.png,True,G01018,train,d6ca5f0e9fca00ff73372aecd3b23627,c57c464b480d3d3f
COVID-1939.png,COVID,COVID/images/COVID-1939.png,COVID/masks/COVID-1939.png,True,G01019,test,cae19f8899a9703e8fc8f080712de3c0,9d2a835b08596b77
COVID-194.png,COVID,COVID/images/COVID-194.png,COVID/masks/COVID-194.png,True,G01020,train,bd75aa058d2c5483ec3f8a159e0d0381,d07d470a2a2d3d5e
COVID-1940.png,COVID,COVID/images/COVID-1940.png,COVID/masks/COVID-1940.png,True,G01021,test,131323482c56534c62f627fce2469447,9b7ad0161a1b393b
COVID-1941.png,COVID,COVID/images/COVID-1941.png,COVID/masks/COVID-1941.png,True,G01022,train,64ed5533a7bbd9cab2ef0cc7d8586ecb,890e861b594d5f5e
COVID-1942.png,COVID,COVID/images/COVID-1942.png,COVID/masks/COVID-1942.png,True,G01023,val,6c34d73bf97c2533017324459196bc1e,d4286bc32fc52f94
COVID-1943.png,COVID,COVID/images/COVID-1943.png,COVID/masks/COVID-1943.png,True,G01024,test,d0ac3ac4f79f1885d6039dc812989a68,dc0a5f1a28f3b2a3
COVID-1944.png,COVID,COVID/images/COVID-1944.png,COVID/masks/COVID-1944.png,True,G01025,train,a4485c471c384cb558dc0e6c28f84638,c97c46025f5f5a98
COVID-1945.png,COVID,COVID/images/COVID-1945.png,COVID/masks/COVID-1945.png,True,G01026,train,ccdeda454115a1b0d9cdccb1e30bf225,817d56025fce3a59
COVID-1946.png,COVID,COVID/images/COVID-1946.png,COVID/masks/COVID-1946.png,True,G01027,train,3410283aeb610af55a852336906cb369,817f961a194d5b1d
COVID-1947.png,COVID,COVID/images/COVID-1947.png,COVID/masks/COVID-1947.png,True,G01028,train,4189c15f28114eb6b0deeb8af0e96bc1,cc3dc30b574b1d4c
COVID-1948.png,COVID,COVID/images/COVID-1948.png,COVID/masks/COVID-1948.png,True,G01029,train,1c15f019c23bcc450a03714e38299bbd,947f532f524e2b44
COVID-1949.png,COVID,COVID/images/COVID-1949.png,COVID/masks/COVID-1949.png,True,G01030,train,b0fbbd32bb7d51c359d584ef129d7ff5,817e87185d5f1a55
COVID-195.png,COVID,COVID/images/COVID-195.png,COVID/masks/COVID-195.png,True,G01031,train,eaebda65a9b3691b3d5e9a8ea1378b1d,d53e42511f57303d
COVID-1950.png,COVID,COVID/images/COVID-1950.png,COVID/masks/COVID-1950.png,True,G01032,train,ada3a2bea6950687fe5f04102e54d2b1,f97c861a9d989994
COVID-1951.png,COVID,COVID/images/COVID-1951.png,COVID/masks/COVID-1951.png,True,G01033,train,886c9babfefed775a68a092856732ee8,817b061a5f797a58
COVID-1952.png,COVID,COVID/images/COVID-1952.png,COVID/masks/COVID-1952.png,True,G01034,train,db64ec94899d760960b6f2dfaa2e8a5a,d13c46061b5eef1a
COVID-1953.png,COVID,COVID/images/COVID-1953.png,COVID/masks/COVID-1953.png,True,G01035,train,c4a4e643049c098c9e39be38d5392892,997bc61a18597b38
COVID-1954.png,COVID,COVID/images/COVID-1954.png,COVID/masks/COVID-1954.png,True,G01036,train,c6e2eb64899fdfafa6cb1786f2f7954c,c42d57d3211d9b53
COVID-1955.png,COVID,COVID/images/COVID-1955.png,COVID/masks/COVID-1955.png,True,G01037,train,29eb684bbd6951f96721006b309a80d0,856c465b6ce769a4
COVID-1956.png,COVID,COVID/images/COVID-1956.png,COVID/masks/COVID-1956.png,True,G01038,train,dd3b80da0c3d7f72a65d8344a90b2463,9968861e296f39f1
COVID-1957.png,COVID,COVID/images/COVID-1957.png,COVID/masks/COVID-1957.png,True,G01039,val,d8a8da018e808910ba5e81b283313546,897d865a0b5d185f
COVID-1958.png,COVID,COVID/images/COVID-1958.png,COVID/masks/COVID-1958.png,True,G01040,val,84108e1c47979b505a1d04e9374454de,817f06061e5f5b1b
COVID-1959.png,COVID,COVID/images/COVID-1959.png,COVID/masks/COVID-1959.png,True,G01041,val,23150ec02d8940c889fac904fb944d21,dc2c470f4c6ea87c
COVID-196.png,COVID,COVID/images/COVID-196.png,COVID/masks/COVID-196.png,True,G01042,test,88e8e31cca70772491bb7e5aa110f531,8d64d01bc31b5be5
COVID-1960.png,COVID,COVID/images/COVID-1960.png,COVID/masks/COVID-1960.png,True,G01043,test,11e470187d0dc0abc27a619128560c27,d03f2e396ac36bc0
COVID-1961.png,COVID,COVID/images/COVID-1961.png,COVID/masks/COVID-1961.png,True,G01044,test,611503e80759219ad55ca52394ef69f9,81b96eda3ad07ad0
COVID-1962.png,COVID,COVID/images/COVID-1962.png,COVID/masks/COVID-1962.png,True,G01045,train,cb8f6cae52dde6dc53405864b545802b,d039473f183e534b
COVID-1963.png,COVID,COVID/images/COVID-1963.png,COVID/masks/COVID-1963.png,True,G01046,train,99fefa7e683d54c09391d8954de01732,c26ac36de84dac95
COVID-1964.png,COVID,COVID/images/COVID-1964.png,COVID/masks/COVID-1964.png,True,G01047,train,0789465e15f13c6d85690c36751210d6,997f05871f0a071f
COVID-1965.png,COVID,COVID/images/COVID-1965.png,COVID/masks/COVID-1965.png,True,G01048,train,cff17e879d475ca731e4929be99411e0,816a0e111f1f3b5f
COVID-1966.png,COVID,COVID/images/COVID-1966.png,COVID/masks/COVID-1966.png,True,G01049,train,349e589afbc18fa7ec3985ae8d504949,d92ac61a5d4b5b49
COVID-1967.png,COVID,COVID/images/COVID-1967.png,COVID/masks/COVID-1967.png,True,G01050,train,78184d60399793ceb73f12219f2dfad0,942e03d70b855f8f
COVID-1968.png,COVID,COVID/images/COVID-1968.png,COVID/masks/COVID-1968.png,True,G01051,train,39021a770dfcbe9b1170f8b60b781c27,846f87d70b4d0f51
COVID-1969.png,COVID,COVID/images/COVID-1969.png,COVID/masks/COVID-1969.png,True,G01052,train,504fc23e98b0cc1b9c7bfb92c1a58423,987dc652154f1b59
COVID-197.png,COVID,COVID/images/COVID-197.png,COVID/masks/COVID-197.png,True,G01053,train,4ee8b7749a64a350c13930f7647c6fcb,997947961ec55a91
COVID-1970.png,COVID,COVID/images/COVID-1970.png,COVID/masks/COVID-1970.png,True,G01054,train,38054c392147e94d56be27bda27ddab9,dd2ec6097d0d3c4c
COVID-1971.png,COVID,COVID/images/COVID-1971.png,COVID/masks/COVID-1971.png,True,G01055,train,4834f38f3655efffab601abaf0965bb0,d938f40397c25397
COVID-1972.png,COVID,COVID/images/COVID-1972.png,COVID/masks/COVID-1972.png,True,G01056,train,010890ad6873a8ef6b869136555f14d7,d82fc65a4f4d584c
COVID-1973.png,COVID,COVID/images/COVID-1973.png,COVID/masks/COVID-1973.png,True,G01057,train,ad3a9e764eda4f6ba27db93a009d166e,997fc61a4d5a184d
COVID-1974.png,COVID,COVID/images/COVID-1974.png,COVID/masks/COVID-1974.png,True,G01058,test,a120f47a472ab2488fda1ac1d9b640a0,dd2a431b474f0e4d
COVID-1975.png,COVID,COVID/images/COVID-1975.png,COVID/masks/COVID-1975.png,True,G01059,train,e6300fca02967544109e19f82c71b0b0,dc38530f5e681c7c
COVID-1976.png,COVID,COVID/images/COVID-1976.png,COVID/masks/COVID-1976.png,True,G01060,train,6508948994a0a7d8dec68e553c17ca31,d50a514b0f6e4e2f
COVID-1977.png,COVID,COVID/images/COVID-1977.png,COVID/masks/COVID-1977.png,True,G01061,val,2ca3839711b47954fe45e948995d80d2,cd2a928fb8a9f0b8
COVID-1978.png,COVID,COVID/images/COVID-1978.png,COVID/masks/COVID-1978.png,True,G01062,test,ee52de7ebb9d092e259828e4b5165fc9,807f87521ccf5e19
COVID-1979.png,COVID,COVID/images/COVID-1979.png,COVID/masks/COVID-1979.png,True,G01063,val,677ba927ec843f3933750db5972ca93c,887f8756144d5e1d
COVID-198.png,COVID,COVID/images/COVID-198.png,COVID/masks/COVID-198.png,True,G01064,train,79048507e3e1bc68235699a4e382e494,816846d67acf7899
COVID-1980.png,COVID,COVID/images/COVID-1980.png,COVID/masks/COVID-1980.png,True,G01065,val,52e3b167474d7337278919e7646f8edf,896ec258cf69e4f0
COVID-1981.png,COVID,COVID/images/COVID-1981.png,COVID/masks/COVID-1981.png,True,G01066,train,f1baa4feaf415f7ab32d1b0299628ab4,dc13430e4b2dadbc
COVID-1982.png,COVID,COVID/images/COVID-1982.png,COVID/masks/COVID-1982.png,True,G01067,test,5db8ac636829a59ec8f2c40090443e1a,897b85471695471f
COVID-1983.png,COVID,COVID/images/COVID-1983.png,COVID/masks/COVID-1983.png,True,G01068,train,ac6086997885a8e20a1391559d4de977,d52ad74b1a47431e
COVID-1984.png,COVID,COVID/images/COVID-1984.png,COVID/masks/COVID-1984.png,True,G01069,train,0043b28974b7d39b33ec66eb4d11a321,d47c039319ca6dcb
COVID-1985.png,COVID,COVID/images/COVID-1985.png,COVID/masks/COVID-1985.png,True,G01070,train,9a082efea7bd63f08a022771072dc7d3,986bc71a0a2b3d2f
COVID-1986.png,COVID,COVID/images/COVID-1986.png,COVID/masks/COVID-1986.png,True,G01071,train,b6ee18f99b8d96f90be84d5ad1d4c27f,917f141a5b1f390e
COVID-1987.png,COVID,COVID/images/COVID-1987.png,COVID/masks/COVID-1987.png,True,G01072,train,9ec06544f754ee0a3654ba62df06b884,81798612175f7a97
COVID-1988.png,COVID,COVID/images/COVID-1988.png,COVID/masks/COVID-1988.png,True,G01073,test,ec3519cd8e9eb79853173491e6305542,d46d3ec34b191d4c
COVID-1989.png,COVID,COVID/images/COVID-1989.png,COVID/masks/COVID-1989.png,True,G01074,train,438c53360c14d1a78b588ed99688794c,817fd0123b39b4bc
COVID-199.png,COVID,COVID/images/COVID-199.png,COVID/masks/COVID-199.png,True,G01075,train,24092fbfa0ec8e6bb338d84b319e5eff,837a041e5b131f5f
COVID-1990.png,COVID,COVID/images/COVID-1990.png,COVID/masks/COVID-1990.png,True,G01076,train,ad944f37675026dcaf945d7dc70673f4,95781e5f7c682263
COVID-1991.png,COVID,COVID/images/COVID-1991.png,COVID/masks/COVID-1991.png,True,G01077,train,5da2faa5872634720f863298657b5532,d42e5dcb0b0e2c3e
COVID-1992.png,COVID,COVID/images/COVID-1992.png,COVID/masks/COVID-1992.png,True,G01078,train,8e9a0e4ad96ff0fca4494b43fa8a207b,d42cd345c3694f6a
COVID-1993.png,COVID,COVID/images/COVID-1993.png,COVID/masks/COVID-1993.png,True,G01079,train,86b0711de02981bff8a3eff5d2f555e8,c47d9353a9c2559a
COVID-1994.png,COVID,COVID/images/COVID-1994.png,COVID/masks/COVID-1994.png,True,G01080,train,49d17bf830353367a4dff2048d64654e,817bc782524d1e5f
COVID-1995.png,COVID,COVID/images/COVID-1995.png,COVID/masks/COVID-1995.png,True,G01081,val,a842b3430ac84d58fedab30f9e363971,c13d46c35c4c4f5d
COVID-1996.png,COVID,COVID/images/COVID-1996.png,COVID/masks/COVID-1996.png,True,G01082,val,329f9544160992878bb4f0b134253981,807f171618595f1f
COVID-1997.png,COVID,COVID/images/COVID-1997.png,COVID/masks/COVID-1997.png,True,G01083,val,79d499cdb353260835293cd10477b706,837a8d101b5b5f1b
COVID-1998.png,COVID,COVID/images/COVID-1998.png,COVID/masks/COVID-1998.png,True,G01084,test,9ede06e229116516d0416f76db72dd96,817aef8514198d5f
COVID-1999.png,COVID,COVID/images/COVID-1999.png,COVID/masks/COVID-1999.png,True,G01085,test,89d8c74db7840c919556991796f482c4,d43c14e2c31bf5c6
COVID-2.png,COVID,COVID/images/COVID-2.png,COVID/masks/COVID-2.png,True,G01086,train,eba33d9078fb72898901dfc4efb63d8d,cd3f60721bc13d1c
COVID-20.png,COVID,COVID/images/COVID-20.png,COVID/masks/COVID-20.png,True,G01087,train,e52fa582f11088396138cacb52719b49,d42ee0a2dbc2f712
COVID-200.png,COVID,COVID/images/COVID-200.png,COVID/masks/COVID-200.png,True,G01088,train,220bf1699ebbb8aeb49111112d2637aa,d47d430b0d2d2d3d
COVID-2000.png,COVID,COVID/images/COVID-2000.png,COVID/masks/COVID-2000.png,True,G01089,train,319db74d1d895f55cfd11e89624f616e,fd2ad70a55505d46
COVID-2001.png,COVID,COVID/images/COVID-2001.png,COVID/masks/COVID-2001.png,True,G01090,train,4d8ca18b3f67bfbde6b2efb73fd8a1d4,dd2a530d4c594f53
COVID-2002.png,COVID,COVID/images/COVID-2002.png,COVID/masks/COVID-2002.png,True,G01091,train,a39d6001cb3626bfc7d9372e3a1ee295,817f84141b1f1b5f
COVID-2003.png,COVID,COVID/images/COVID-2003.png,COVID/masks/COVID-2003.png,True,G01092,train,fa5e2d91804f3e760bd648db9d6290bf,bc2ad38b451e4dc5
COVID-2004.png,COVID,COVID/images/COVID-2004.png,COVID/masks/COVID-2004.png,True,G01093,test,9658b9548c89826f89fc14e50427405a,8179061e1a575f5d
COVID-2005.png,COVID,COVID/images/COVID-2005.png,COVID/masks/COVID-2005.png,True,G01094,train,c16fcdd7ee063ac0485c4f45bfd86e5e,dd2c50d21b4f4d56
COVID-2006.png,COVID,COVID/images/COVID-2006.png,COVID/masks/COVID-2006.png,True,G01095,train,c00c6636c8fd669821cca10dbe2c08fe,813f065e58497d5d
COVID-2007.png,COVID,COVID/images/COVID-2007.png,COVID/masks/COVID-2007.png,True,G01096,train,a22bbba1f4efd1a7d0704b9b904d1fab,ac2a830f5f4b4f43
COVID-2008.png,COVID,COVID/images/COVID-2008.png,COVID/masks/COVID-2008.png,True,G01097,val,f21078e449e9d416816b882f11b17240,9177153a2a3b2a3b
COVID-2009.png,COVID,COVID/images/COVID-2009.png,COVID/masks/COVID-2009.png,True,G01098,val,b477e9136d1932e4a1943cba4378e443,d52f46175a854f15
COVID-201.png,COVID,COVID/images/COVID-201.png,COVID/masks/COVID-201.png,True,G01099,train,512e15300e6098b1a886d93ef4d61e51,d979e51785c01ee8
COVID-2010.png,COVID,COVID/images/COVID-2010.png,COVID/masks/COVID-2010.png,True,G01100,train,54d52b769d1ed5698d159c044c6bd296,d52ac2438fc3591f
COVID-2011.png,COVID,COVID/images/COVID-2011.png,COVID/masks/COVID-2011.png,True,G01101,test,8bd9697ab90a822bff3c4ef9e3c80928,d52e43434f4d4c4f
COVID-2012.png,COVID,COVID/images/COVID-2012.png,COVID/masks/COVID-2012.png,True,G01102,train,841c6bb72e5ec7f31b3553a428da71db,d4bdc2c3dfc01e0c
COVID-2013.png,COVID,COVID/images/COVID-2013.png,COVID/masks/COVID-2013.png,True,G01103,train,a91340e89bde2d6edbefeda285f9fd71,882dc30f7c6cf4e1
COVID-2014.png,COVID,COVID/images/COVID-2014.png,COVID/masks/COVID-2014.png,True,G01104,train,e972c68e610b27da21ecd7282b031077,81787a568a5a5b5b
COVID-2015.png,COVID,COVID/images/COVID-2015.png,COVID/masks/COVID-2015.png,True,G01105,train,91cb1062542d9fad2def0d67358d1d27,cca7a183d5455f4a
COVID-2016.png,COVID,COVID/images/COVID-2016.png,COVID/masks/COVID-2016.png,True,G01106,train,fe1aec8950496573cef22842673f3d46,ccafa183d5455f48
COVID-2017.png,COVID,COVID/images/COVID-2017.png,COVID/masks/COVID-2017.png,True,G01107,val,60693f64f29eb30937a2cb005d967769,98af879dc5d4d0c2
COVID-2018.png,COVID,COVID/images/COVID-2018.png,COVID/masks/COVID-2018.png,True,G01108,test,7bf4f20f0e35a9329c836f1dddbd2b28,817ae4dfdc945819
COVID-2019.png,COVID,COVID/images/COVID-2019.png,COVID/masks/COVID-2019.png,True,G01109,train,24cc3be1d865aa156f8bc3d0334ceee8,817c1e7a6765e464
COVID-202.png,COVID,COVID/images/COVID-202.png,COVID/masks/COVID-202.png,True,G01110,test,b11d27b8dcbb42a83b958365531c2fa4,817946161a5d5f5b
COVID-2020.png,COVID,COVID/images/COVID-2020.png,COVID/masks/COVID-2020.png,True,G01111,train,1b8274ec8fea210ccb88998cca964c12,dc35e3c38ec2c30b
COVID-2021.png,COVID,COVID/images/COVID-2021.png,COVID/masks/COVID-2021.png,True,G01112,train,5386d919e10da47c1f4d9bcb39a5c2f7,c946d769a1baa4b8
COVID-2022.png,COVID,COVID/images/COVID-2022.png,COVID/masks/COVID-2022.png,True,G01113,train,218e2ec1bd4f16b12b176ec381db56c5,996f871ec638c239
COVID-2023.png,COVID,COVID/images/COVID-2023.png,COVID/masks/COVID-2023.png,True,G01114,train,2c06a853ddcf7c2593f624b6e9152ec2,917a141a5b7f788c
COVID-2024.png,COVID,COVID/images/COVID-2024.png,COVID/masks/COVID-2024.png,True,G01115,train,a819585ba770365953a88d9f6dfaca94,d43d42555d4a4b4f
COVID-2025.png,COVID,COVID/images/COVID-2025.png,COVID/masks/COVID-2025.png,True,G01116,train,f9f4f6b4892b298de0909cfe2d2ea519,956c075b3ae27ae0
COVID-2026.png,COVID,COVID/images/COVID-2026.png,COVID/masks/COVID-2026.png,True,G01117,val,7c60d5c4a40d55b9de349b22d96d9c6f,817f061e1d1d1e5d
COVID-2027.png,COVID,COVID/images/COVID-2027.png,COVID/masks/COVID-2027.png,True,G01118,train,13c5113b07bf98abdbbb00d5c4e6356a,d52a530e323db572
COVID-2028.png,COVID,COVID/images/COVID-2028.png,COVID/masks/COVID-2028.png,True,G01119,train,a7bb31ee8dc78ee53cb897eead8b3984,803f165e6be16fa0
COVID-2029.png,COVID,COVID/images/COVID-2029.png,COVID/masks/COVID-2029.png,True,G01120,train,5ede53da1b94da087ef0f04fab044d01,9138070f0d4f5c7f
COVID-203.png,COVID,COVID/images/COVID-203.png,COVID/masks/COVID-203.png,True,G01121,val,b9581f28c0be551231ead4beb4ac5e89,9d2fc6131b474f18
COVID-2030.png,COVID,COVID/images/COVID-2030.png,COVID/masks/COVID-2030.png,True,G01122,test,83533e99ccd1235abcd7150dcadc3f9d,91794e0c195f5e78
COVID-2031.png,COVID,COVID/images/COVID-2031.png,COVID/masks/COVID-2031.png,True,G01123,val,e4682d3f97c708d6193ed5706893300e,817f87161a4d5f19
COVID-2032.png,COVID,COVID/images/COVID-2032.png,COVID/masks/COVID-2032.png,True,G01124,test,5945eaf4bc4fa8ba05c470070cc19b8e,d42e434b1b471e1f
COVID-2033.png,COVID,COVID/images/COVID-2033.png,COVID/masks/COVID-2033.png,True,G01125,train,0bce636e9f536b58e27188455f25345b,9d39c71a091f194f
COVID-2034.png,COVID,COVID/images/COVID-2034.png,COVID/masks/COVID-2034.png,True,G01126,train,c1a0fcb2bd4ac3dbcaae468aa72ee493,cc2f435d6c647ae0
COVID-2035.png,COVID,COVID/images/COVID-2035.png,COVID/masks/COVID-2035.png,True,G01127,train,e8994cb65824c127a6e68b63e7295328,dc2c42c31b4f3e27
COVID-2036.png,COVID,COVID/images/COVID-2036.png,COVID/masks/COVID-2036.png,True,G01128,train,2e1934b5d9e7774eb1924f3c32cdcc93,c03d56920b1f9f0f
COVID-2037.png,COVID,COVID/images/COVID-2037.png,COVID/masks/COVID-2037.png,True,G01129,train,5fb847e56e7b6353798a5a6036af84a5,d42e07591d4e7e1c
COVID-2038.png,COVID,COVID/images/COVID-2038.png,COVID/masks/COVID-2038.png,True,G01130,train,6b35e4013751810e8b6670ad34448de2,d53dc28b1f424d1d
COVID-2039.png,COVID,COVID/images/COVID-2039.png,COVID/masks/COVID-2039.png,True,G01131,train,5f8c31fbace5832cbd7001d7c3aba95c,817c8e1a155e595f
COVID-204.png,COVID,COVID/images/COVID-204.png,COVID/masks/COVID-204.png,True,G01132,train,595ab3e234a3f6a0ec04d3dd3f44b244,dd28520b1f4f0f3c
COVID-2040.png,COVID,COVID/images/COVID-2040.png,COVID/masks/COVID-2040.png,True,G01133,train,dba1564f773efd0a0fe1805016d8e8f0,817d845e975f1e12
COVID-2041.png,COVID,COVID/images/COVID-2041.png,COVID/masks/COVID-2041.png,True,G01134,train,f5311029048d7763295d2a1cbe4d7a6f,997fd2584f58180f
COVID-2042.png,COVID,COVID/images/COVID-2042.png,COVID/masks/COVID-2042.png,True,G01135,train,67598fe43b1fe61243cdf39692cabdf1,991d87561a5b1d5a
COVID-2043.png,COVID,COVID/images/COVID-2043.png,COVID/masks/COVID-2043.png,True,G01136,val,067802e523059419783118e2566e41c1,bc2ec71b1f420e5c
COVID-2044.png,COVID,COVID/images/COVID-2044.png,COVID/masks/COVID-2044.png,True,G01137,val,b0a5a6960751a6102df5f6d25d84fd0b,882f461add45f555
COVID-2045.png,COVID,COVID/images/COVID-2045.png,COVID/masks/COVID-2045.png,True,G01138,test,841ae1074827c74a8ef61811ee322290,dc2d821f1d86dd1c
COVID-2046.png,COVID,COVID/images/COVID-2046.png,COVID/masks/COVID-2046.png,True,G01139,train,981002d814e2d28c5782eba5e11b12a9,d5b502da2ad22bdc
COVID-2047.png,COVID,COVID/images/COVID-2047.png,COVID/masks/COVID-2047.png,True,G01140,val,7d35c8c2d4cdf60adaf640439fe4a3c0,b92f86da88c6da49
COVID-2048.png,COVID,COVID/images/COVID-2048.png,COVID/masks/COVID-2048.png,True,G01141,test,6bf4b43f9267542d564a433c93f02121,9928869e1c5dba5d
COVID-2049.png,COVID,COVID/images/COVID-2049.png,COVID/masks/COVID-2049.png,True,G01142,test,92d615a8145df85b93e52fea98dfd95b,dc0f430deaf59283
COVID-205.png,COVID,COVID/images/COVID-205.png,COVID/masks/COVID-205.png,True,G01143,train,763c3b1406e74d98ead3915446660355,b863c71ec319bd41
COVID-2050.png,COVID,COVID/images/COVID-2050.png,COVID/masks/COVID-2050.png,True,G01144,train,457685a64934481146631f7528779a3c,903f461e584d5f1e
COVID-2051.png,COVID,COVID/images/COVID-2051.png,COVID/masks/COVID-2051.png,True,G01145,train,57f09f77e438a962cc68b766ea961d76,997b521849393d2f
COVID-2052.png,COVID,COVID/images/COVID-2052.png,COVID/masks/COVID-2052.png,True,G01146,val,593ca0058e45409c7ac5e978a3fceb9a,911b141a593b7aeb
COVID-2053.png,COVID,COVID/images/COVID-2053.png,COVID/masks/COVID-2053.png,True,G01147,train,a95aefbbc194ded6ecb5d9dfa4c51b61,995f061a187b38f9
COVID-2054.png,COVID,COVID/images/COVID-2054.png,COVID/masks/COVID-2054.png,True,G01148,train,12588d2c02d77b3149df84e404ab467d,d1ffd65840795548
COVID-2055.png,COVID,COVID/images/COVID-2055.png,COVID/masks/COVID-2055.png,True,G01149,train,15b5022272c378b323ba9ccb843dd90a,d12b861e181d1f5f
COVID-2056.png,COVID,COVID/images/COVID-2056.png,COVID/masks/COVID-2056.png,True,G01150,test,8a582bfbc6dfd039cbb0cf2aea1aaf35,dd2a42435f4b0f5c
COVID-2057.png,COVID,COVID/images/COVID-2057.png,COVID/masks/COVID-2057.png,True,G01151,train,a53bd9007428c6cca4f87fe4ea8494ab,817a56481b6b7d1b
COVID-2058.png,COVID,COVID/images/COVID-2058.png,COVID/masks/COVID-2058.png,True,G01152,train,510eb64c9b269ce2ea5b25493ca071dc,917947d31b494f1a
COVID-2059.png,COVID,COVID/images/COVID-2059.png,COVID/masks/COVID-2059.png,True,G01153,train,3ef359fe9dd45be20b2c131536c9e9fc,817f07061e595d5d
COVID-206.png,COVID,COVID/images/COVID-206.png,COVID/masks/COVID-206.png,True,G01154,train,701686ce58c0243b00d620f374ae52b5,dc34434b434b1f4f
COVID-2060.png,COVID,COVID/images/COVID-2060.png,COVID/masks/COVID-2060.png,True,G01155,train,8cac0ee69a548ea938b831cb3b9dad25,d53c131317574b55
COVID-2061.png,COVID,COVID/images/COVID-2061.png,COVID/masks/COVID-2061.png,True,G01156,train,0eb7329e5665eae4f32f8f69ebcaaf36,984b1a596dbce3b0
COVID-2062.png,COVID,COVID/images/COVID-2062.png,COVID/masks/COVID-2062.png,True,G01157,train,7c67e38d8e1f621f83aebf54ef0bb2f4,d52c1f0b1f1a4e4e
COVID-2063.png,COVID,COVID/images/COVID-2063.png,COVID/masks/COVID-2063.png,True,G01158,train,ab0b6d07228be500f8a9b134affd586f,91391e18595d7c7a
COVID-2064.png,COVID,COVID/images/COVID-2064.png,COVID/masks/COVID-2064.png,True,G01159,train,da7c11d32cd11a23bcf1a45797e32eb3,d52c570b5f124b4b
COVID-2065.png,COVID,COVID/images/COVID-2065.png,COVID/masks/COVID-2065.png,True,G01160,val,8ca79e0470ac4bb56036538b5649823a,9160169e1f1e5e5b
COVID-2066.png,COVID,COVID/images/COVID-2066.png,COVID/masks/COVID-2066.png,True,G01161,val,fb59d081c13e949a34cbb5bed42b1851,95281f5e1a5e1a7a
COVID-2067.png,COVID,COVID/images/COVID-2067.png,COVID/masks/COVID-2067.png,True,G01162,train,d6a57ec518b0c17f8db5e7294a39393d,987b071e181b3be5
COVID-2068.png,COVID,COVID/images/COVID-2068.png,COVID/masks/COVID-2068.png,True,G01163,test,039f569768715a6a7cfe7d0be3c9e8ed,997fc65e10565954
COVID-2069.png,COVID,COVID/images/COVID-2069.png,COVID/masks/COVID-2069.png,True,G01164,test,512b5f3ebd066df12930c06accab6721,817d86161a5ac75f
COVID-207.png,COVID,COVID/images/COVID-207.png,COVID/masks/COVID-207.png,True,G01165,train,23f907f98016e861c289226dc28e6684,d939e51f85c01ee8
COVID-2070.png,COVID,COVID/images/COVID-2070.png,COVID/masks/COVID-2070.png,True,G01166,train,e89d0c348da8d399f26f7e1ae2b0d2f2,d42f435d0d0e2f0f
COVID-2071.png,COVID,COVID/images/COVID-2071.png,COVID/masks/COVID-2071.png,True,G01167,train,ced71a9a8a3a0850851c9ab386f6cd61,d03dc2971a4f5b1c
COVID-2072.png,COVID,COVID/images/COVID-2072.png,COVID/masks/COVID-2072.png,True,G01168,train,d85104034429b4766304b403a8f5c947,c84d47161e4f5d1e
COVID-2073.png,COVID,COVID/images/COVID-2073.png,COVID/masks/COVID-2073.png,True,G01169,train,50137721d3090c132baed85beaa7be23,d43e51494d4b2b4f
COVID-2074.png,COVID,COVID/images/COVID-2074.png,COVID/masks/COVID-2074.png,True,G01170,train,4e2dc65b9d125e32c8c13990fb0a5051,806dc7921b4f4e5b
COVID-2075.png,COVID,COVID/images/COVID-2075.png,COVID/masks/COVID-2075.png,True,G01171,train,8669aa862f5db31cdc6c7b8afc7589dc,d95e1e59e9a9a501
COVID-2076.png,COVID,COVID/images/COVID-2076.png,COVID/masks/COVID-2076.png,True,G01172,train,9f2809e084134b52aa73b4a3e5d579eb,907d46535d447f1c
COVID-2077.png,COVID,COVID/images/COVID-2077.png,COVID/masks/COVID-2077.png,True,G01173,train,2aa28a052e51c5c95943c2072c4b3f32,886e065a0d7c7ced
COVID-2078.png,COVID,COVID/images/COVID-2078.png,COVID/masks/COVID-2078.png,True,G01174,train,69bf625aba66de8bd444234a9a84d43c,dc1f431f4c291d8b
COVID-2079.png,COVID,COVID/images/COVID-2079.png,COVID/masks/COVID-2079.png,True,G01175,train,f3f64ab8e37f521fa1c3d8c113c618a6,d17b561a54435e5a
COVID-208.png,COVID,COVID/images/COVID-208.png,COVID/masks/COVID-208.png,True,G01176,train,244a500d246faca43bc1f9032deba253,dd22531f4a1d463e
COVID-2080.png,COVID,COVID/images/COVID-2080.png,COVID/masks/COVID-2080.png,True,G01177,val,f5cb2f1095c5e2956c7a3adcd8bbeaf6,d42a531a4a2b3eaf
COVID-2081.png,COVID,COVID/images/COVID-2081.png,COVID/masks/COVID-2081.png,True,G01178,val,516482f1c59e87e0ff4190c9dbfb4f2e,b96e0679e5e4b818
COVID-2082.png,COVID,COVID/images/COVID-2082.png,COVID/masks/COVID-2082.png,True,G01179,val,7583bff7a8710c663a96cbb14bfaf326,f96947561a474738
COVID-2083.png,COVID,COVID/images/COVID-2083.png,COVID/masks/COVID-2083.png,True,G01180,train,38a5ecf3ae9fb0b96c020d0717f61ed6,9c2643494c6e6eeb
COVID-2084.png,COVID,COVID/images/COVID-2084.png,COVID/masks/COVID-2084.png,True,G01181,test,0cdd53b939b058525c75486a7333fdcb,b92e46586575e4b8
COVID-2085.png,COVID,COVID/images/COVID-2085.png,COVID/masks/COVID-2085.png,True,G01181,test,0cdd53b939b058525c75486a7333fdcb,b92e46586575e4b8
COVID-2086.png,COVID,COVID/images/COVID-2086.png,COVID/masks/COVID-2086.png,True,G01182,test,4371b05f4328e589fae18575dddd7583,d42e434c6ce3d3c3
COVID-2087.png,COVID,COVID/images/COVID-2087.png,COVID/masks/COVID-2087.png,True,G01183,train,1b3bd1cc4289186ceebad7aca4a1f0e2,997dc61b1a455e19
COVID-2088.png,COVID,COVID/images/COVID-2088.png,COVID/masks/COVID-2088.png,True,G01184,train,407e8d8039847b405d14cb1eac35cf07,d52bc30f1e682f23
COVID-2089.png,COVID,COVID/images/COVID-2089.png,COVID/masks/COVID-2089.png,True,G01185,train,baaf853a08e34d7a4a769f33981ae877,807bd78e1a186b3b
COVID-209.png,COVID,COVID/images/COVID-209.png,COVID/masks/COVID-209.png,True,G01186,test,5628f78926b2f6fc5ec0bb03218ea73e,903d4ec71a1b696d
COVID-2090.png,COVID,COVID/images/COVID-2090.png,COVID/masks/COVID-2090.png,True,G01187,train,6193a0bd4ae2edf2889cf58a0c7c7b0d,dd2a735b2b2c20ad
COVID-2091.png,COVID,COVID/images/COVID-2091.png,COVID/masks/COVID-2091.png,True,G01188,train,27784ea3190e1ca5d13fc3fd7a6117b3,dc3d43169ec35a68
COVID-2092.png,COVID,COVID/images/COVID-2092.png,COVID/masks/COVID-2092.png,True,G01189,train,da8f4ddecd7ed7d26c9132fd6fdfb463,9b6886948a8f399f
COVID-2093.png,COVID,COVID/images/COVID-2093.png,COVID/masks/COVID-2093.png,True,G01190,train,efc67e8a0f050bd3dd1060f1ea4d597c,802987969e4f3d4f
COVID-2094.png,COVID,COVID/images/COVID-2094.png,COVID/masks/COVID-2094.png,True,G01191,train,70069d71ad346c042c5936a278bd3ae6,912946cf5259eb56
COVID-2095.png,COVID,COVID/images/COVID-2095.png,COVID/masks/COVID-2095.png,True,G01192,train,43b2685345ac16fede2bfd08daf4753d,996f861a094f587d
COVID-2096.png,COVID,COVID/images/COVID-2096.png,COVID/masks/COVID-2096.png,True,G01193,train,180b80ab61a98843a96a782897449879,ec0ac3cbcc8bbda0
COVID-2097.png,COVID,COVID/images/COVID-2097.png,COVID/masks/COVID-2097.png,True,G01194,train,258b3446796d592eb7e3da4413d452d7,807d47025f4f4b5b
COVID-2098.png,COVID,COVID/images/COVID-2098.png,COVID/masks/COVID-2098.png,True,G01195,train,52f1694724a978bbc97b8d766c821d36,d87c46435b4d4b1b
COVID-2099.png,COVID,COVID/images/COVID-2099.png,COVID/masks/COVID-2099.png,True,G01196,train,e61a89ac24b89658e7b52633d1d94858,d43f425d1a6c3c6c
COVID-21.png,COVID,COVID/images/COVID-21.png,COVID/masks/COVID-21.png,True,G00869,train,a00f1e48eaebd032eee090def7795dac,816c0e1a5f7d7c4c
COVID-210.png,COVID,COVID/images/COVID-210.png,COVID/masks/COVID-210.png,True,G01197,val,55ab0cbc21b40a685afd928495c72baf,dd2ac30b5b470f54
COVID-2100.png,COVID,COVID/images/COVID-2100.png,COVID/masks/COVID-2100.png,True,G01198,train,448fbacf530fb24a671e27edab3de28d,917c474a9bcb4a5a
COVID-2101.png,COVID,COVID/images/COVID-2101.png,COVID/masks/COVID-2101.png,True,G01199,val,d2f24aaa93af59662d78654344760ef3,9d3d821e551f1b0d
COVID-2102.png,COVID,COVID/images/COVID-2102.png,COVID/masks/COVID-2102.png,True,G01200,train,3d3dc6955f183d6e1bf3e58f9f42c881,993bc7868cc68dc6
COVID-2103.png,COVID,COVID/images/COVID-2103.png,COVID/masks/COVID-2103.png,True,G01201,train,ec76657a0b5a3e0b9a2e08ff7f551d02,9d3e84431b5d4f1a
COVID-2104.png,COVID,COVID/images/COVID-2104.png,COVID/masks/COVID-2104.png,True,G01202,train,9878bdad09c1128633da3d5f3024f715,816907161e1f3f1f
COVID-2105.png,COVID,COVID/images/COVID-2105.png,COVID/masks/COVID-2105.png,True,G01203,val,8894ea6e0121dbd7318861c130670166,9c4c471e5c5d78e1
COVID-2106.png,COVID,COVID/images/COVID-2106.png,COVID/masks/COVID-2106.png,True,G01204,test,6633c9f335bc7ac2a4ae7f05980efc0c,d5f806573a82d743
COVID-2107.png,COVID,COVID/images/COVID-2107.png,COVID/masks/COVID-2107.png,True,G01205,test,3fe312124dd6506ad892d92028aaf008,d62fc1d14d4e4d4a
COVID-2108.png,COVID,COVID/images/COVID-2108.png,COVID/masks/COVID-2108.png,True,G01206,test,87b2f39500f69b1c074adf238ac4a7d6,d52843db0bc52add
COVID-2109.png,COVID,COVID/images/COVID-2109.png,COVID/masks/COVID-2109.png,True,G01207,train,028adffb0ddfdb90f1a830782089eb19,9d3947d61c555919
COVID-211.png,COVID,COVID/images/COVID-211.png,COVID/masks/COVID-211.png,True,G01208,train,f6526535895648b81a3fda1fef6f0470,dc4f4b0f0c2cadb4
COVID-2110.png,COVID,COVID/images/COVID-2110.png,COVID/masks/COVID-2110.png,True,G01209,train,ddff955b2a164be7e16419d3b7bd4d2c,dd0ec24b5f4d0a4d
COVID-2111.png,COVID,COVID/images/COVID-2111.png,COVID/masks/COVID-2111.png,True,G01210,test,2d5a41c78bdc1c866c8408b27d682ce7,ac0ec39bf0f4f0b0
COVID-2112.png,COVID,COVID/images/COVID-2112.png,COVID/masks/COVID-2112.png,True,G01211,train,647431ce468590b2e45e4c78948b3171,cc4bc30f2cbaece0
COVID-2113.png,COVID,COVID/images/COVID-2113.png,COVID/masks/COVID-2113.png,True,G01212,train,45c7fa44c2e8caff81cf49df3ec54c29,dc28d2c38f56e247
COVID-2114.png,COVID,COVID/images/COVID-2114.png,COVID/masks/COVID-2114.png,True,G01213,train,0e2f264b9b5e285d48aabfb70cb1d2bd,89ecc6929bc33c78
COVID-2115.png,COVID,COVID/images/COVID-2115.png,COVID/masks/COVID-2115.png,True,G01214,train,92bb4e810ecdb840d471dd9c136159ae,d0d563c20fdad683
COVID-2116.png,COVID,COVID/images/COVID-2116.png,COVID/masks/COVID-2116.png,True,G01215,train,ed4afcbd2755d28a3fa11603857cd695,987d87161a0f1f1e
COVID-2117.png,COVID,COVID/images/COVID-2117.png,COVID/masks/COVID-2117.png,True,G01216,train,2393f374f1a148fa1d3a0ef7a7affbcb,995b56185d383da6
COVID-2118.png,COVID,COVID/images/COVID-2118.png,COVID/masks/COVID-2118.png,True,G01217,val,69e18dfe2c88fb9aababcfe216897872,dc2a530d0e1b6f3a
COVID-2119.png,COVID,COVID/images/COVID-2119.png,COVID/masks/COVID-2119.png,True,G01218,val,60bef4855fcb8a65db5b8d4daddaf3a6,9d28475a4e5c796c
COVID-212.png,COVID,COVID/images/COVID-212.png,COVID/masks/COVID-212.png,True,G01219,test,dc8c004df41d398606ad26b18ffd3284,893f861a1d4e195f
COVID-2120.png,COVID,COVID/images/COVID-2120.png,COVID/masks/COVID-2120.png,True,G01220,test,75a5d68e71142d58eba576fdd91fda96,d42ea3921f3d3c2c
COVID-2121.png,COVID,COVID/images/COVID-2121.png,COVID/masks/COVID-2121.png,True,G01221,train,66afa56b980b45fcf42a56928e872ef0,9b78dff3e0848416
COVID-2122.png,COVID,COVID/images/COVID-2122.png,COVID/masks/COVID-2122.png,True,G01222,train,60f0d1b570e97cb38c66510f57271d5d,9b78cfd3c2848617
COVID-2123.png,COVID,COVID/images/COVID-2123.png,COVID/masks/COVID-2123.png,True,G01223,train,fb9f45be08d474aa23f30e96f35da07a,997946165a847bf8
COVID-2124.png,COVID,COVID/images/COVID-2124.png,COVID/masks/COVID-2124.png,True,G01224,train,d7cf0b3d285dea4ed0e5fdf7d7b69eda,937f04726b7163a4
COVID-2125.png,COVID,COVID/images/COVID-2125.png,COVID/masks/COVID-2125.png,True,G01225,val,0dd5d7d47747ffdd87130da373ff0884,933e8472ebe1e384
COVID-2126.png,COVID,COVID/images/COVID-2126.png,COVID/masks/COVID-2126.png,True,G01226,train,177a5127f0e72481e9abe708313f3aea,c42d434f2a6e6ee4
COVID-2127.png,COVID,COVID/images/COVID-2127.png,COVID/masks/COVID-2127.png,True,G01227,train,40e7dd7fe4639db92254f5c54b7908a9,837a041d7a7a7b43
COVID-2128.png,COVID,COVID/images/COVID-2128.png,COVID/masks/COVID-2128.png,True,G01228,train,af9a89551cd0a030927658678540f82b,c05f421b1d2d3dad
COVID-2129.png,COVID,COVID/images/COVID-2129.png,COVID/masks/COVID-2129.png,True,G01229,train,e22761ef3e5377e5d1c7c90e9416b52d,8c2b134d7472e79a
COVID-213.png,COVID,COVID/images/COVID-213.png,COVID/masks/COVID-213.png,True,G01230,train,94df5163e9f970221c463e7f51eb3315,d52b424b0f2d3c3d
COVID-2130.png,COVID,COVID/images/COVID-2130.png,COVID/masks/COVID-2130.png,True,G01231,train,d02ad42301c6186b2f787aae02031bff,817fc452127f585b
COVID-2131.png,COVID,COVID/images/COVID-2131.png,COVID/masks/COVID-2131.png,True,G01232,train,b118b2e0ea7e2714b3ecd999d906055d,91480e1b79fc78e5
COVID-2132.png,COVID,COVID/images/COVID-2132.png,COVID/masks/COVID-2132.png,True,G01233,val,2c15aef8934fa4e4472f37297faf9b46,917f161a7b783921
COVID-2133.png,COVID,COVID/images/COVID-2133.png,COVID/masks/COVID-2133.png,True,G01234,train,51a85ea6952854a90b5866fd7376a6c0,897f041a5f5a194f
COVID-2134.png,COVID,COVID/images/COVID-2134.png,COVID/masks/COVID-2134.png,True,G01235,train,86fb5317cee3be65cfde1c6e7cda42ec,ac6cc30f4d1c4d5d
COVID-2135.png,COVID,COVID/images/COVID-2135.png,COVID/masks/COVID-2135.png,True,G01236,train,32a007de1a6c5e17f8dc945274649a92,d52c471a1aedb8ac
COVID-2136.png,COVID,COVID/images/COVID-2136.png,COVID/masks/COVID-2136.png,True,G01237,train,3a3922b4d62c38d64458f6ec2898ddd3,d43dd5d30a89df40
COVID-2137.png,COVID,COVID/images/COVID-2137.png,COVID/masks/COVID-2137.png,True,G01238,train,cfea0d26a63404f2c1f67761ed545c75,915947061e195f5f
COVID-2138.png,COVID,COVID/images/COVID-2138.png,COVID/masks/COVID-2138.png,True,G01239,val,d3fb0d944896f82f66104d068363cabb,d42c434f1e4a4e5f
COVID-2139.png,COVID,COVID/images/COVID-2139.png,COVID/masks/COVID-2139.png,True,G01240,val,1a56c95f0b5043ba33929f009d0564cc,997945c61e59591b
COVID-214.png,COVID,COVID/images/COVID-214.png,COVID/masks/COVID-214.png,True,G01241,train,fb680e9b139079108276784ee21e20ec,dc2ec34b0d8a0f2f
COVID-2140.png,COVID,COVID/images/COVID-2140.png,COVID/masks/COVID-2140.png,True,G01242,test,f4dc9d9f0637f3bde88b4c4d38782029,d42c53431e4e4b5f
COVID-2141.png,COVID,COVID/images/COVID-2141.png,COVID/masks/COVID-2141.png,True,G01243,test,dd1a1fdf8729734b317bf5a249ca56c3,d42cd2518bd7c655
COVID-2142.png,COVID,COVID/images/COVID-2142.png,COVID/masks/COVID-2142.png,True,G01244,test,470f3dc3c98de60b4483058e475f84cf,817b0c141f1d1f5f
COVID-2143.png,COVID,COVID/images/COVID-2143.png,COVID/masks/COVID-2143.png,True,G01245,train,1bc607b239ca0d0ec1d19530b78ae748,ac2a030a4d8f5fcf
COVID-2144.png,COVID,COVID/images/COVID-2144.png,COVID/masks/COVID-2144.png,True,G01246,train,cc54e4cd23a752b2a43a02dabb7e5b56,91791f1e386939e2
COVID-2145.png,COVID,COVID/images/COVID-2145.png,COVID/masks/COVID-2145.png,True,G01247,train,9ae02a6b36d36e4349decd2f87173a3c,dc6fd0130f493c39
COVID-2146.png,COVID,COVID/images/COVID-2146.png,COVID/masks/COVID-2146.png,True,G01248,train,7ae5520b0a67fd81dd6ea5a0317e1e9d,d12806165b5e5e5f
COVID-2147.png,COVID,COVID/images/COVID-2147.png,COVID/masks/COVID-2147.png,True,G01249,train,9f78fed4732f6bf95bd0519df4ea2de9,87397c9678947899
COVID-2148.png,COVID,COVID/images/COVID-2148.png,COVID/masks/COVID-2148.png,True,G01250,train,ae56dae0c786a8120e8b17a7ed6e6d91,817f563a69a165c5
COVID-2149.png,COVID,COVID/images/COVID-2149.png,COVID/masks/COVID-2149.png,True,G01251,train,d4c82523cd9d929886848ec33da5b741,847c57531a4d6c7c
COVID-215.png,COVID,COVID/images/COVID-215.png,COVID/masks/COVID-215.png,True,G01252,train,2217adc0ebe170488a179b479ce84a61,d42ed34b49432f1d
COVID-2150.png,COVID,COVID/images/COVID-2150.png,COVID/masks/COVID-2150.png,True,G01253,train,dd20be629302627cac5b9e2aff949912,806d471e5a4a7f5a
COVID-2151.png,COVID,COVID/images/COVID-2151.png,COVID/masks/COVID-2151.png,True,G01254,train,10aedbbca0faefd025297a2960794d1d,81795e1e1a5e3939
COVID-2152.png,COVID,COVID/images/COVID-2152.png,COVID/masks/COVID-2152.png,True,G01255,train,ff7b83b6141c5a2186013513710e15ae,81287e867a937a8f
COVID-2153.png,COVID,COVID/images/COVID-2153.png,COVID/masks/COVID-2153.png,True,G01256,val,badefae4c8670742b96f55babca955cb,dd3dc21b4d441f45
COVID-2154.png,COVID,COVID/images/COVID-2154.png,COVID/masks/COVID-2154.png,True,G01257,val,5fd427e3556703984fe333cc2cab0a28,cd0e065e4cfcf8e0
COVID-2155.png,COVID,COVID/images/COVID-2155.png,COVID/masks/COVID-2155.png,True,G01258,train,2de6f5bff3de96739bed84c7e20d1703,dd2a52430f4e7c2d
COVID-2156.png,COVID,COVID/images/COVID-2156.png,COVID/masks/COVID-2156.png,True,G01259,test,2c4a51dafc6c50456b48614b4aea2604,817b14161a5d5f5b
COVID-2157.png,COVID,COVID/images/COVID-2157.png,COVID/masks/COVID-2157.png,True,G01260,test,730e714392564909493c91cbf2f8101f,9d2cd31bc5545d45
COVID-2158.png,COVID,COVID/images/COVID-2158.png,COVID/masks/COVID-2158.png,True,G01261,train,1c9c36b9b57b9dbbfbecc28cdba0ed34,9c38c393ded3141d
COVID-2159.png,COVID,COVID/images/COVID-2159.png,COVID/masks/COVID-2159.png,True,G01262,train,88750bd4bb930b2b28073377a9653cd1,8979c6961e491f5a
COVID-216.png,COVID,COVID/images/COVID-216.png,COVID/masks/COVID-216.png,True,G01263,train,ea8ac7cae619ff0cdf9893e1ad1c41f0,8e0ec0171fcbc91f
COVID-2160.png,COVID,COVID/images/COVID-2160.png,COVID/masks/COVID-2160.png,True,G01264,train,d93a288e2d9fc8d08d1939329620aa46,91681e14195bff59
COVID-2161.png,COVID,COVID/images/COVID-2161.png,COVID/masks/COVID-2161.png,True,G01265,train,fa91d7da8b0632d948a5b24b3967f0f8,d43e430b4f4c2e6e
COVID-2162.png,COVID,COVID/images/COVID-2162.png,COVID/masks/COVID-2162.png,True,G01266,val,056d69c6e0ad1e69891f4e828278604b,817b578616114f5f
COVID-2163.png,COVID,COVID/images/COVID-2163.png,COVID/masks/COVID-2163.png,True,G01267,train,b1dd370a6ed6f0a816846d4fadb44776,d47e434b584d6e38
COVID-2164.png,COVID,COVID/images/COVID-2164.png,COVID/masks/COVID-2164.png,True,G01268,val,998b3b10c49f857fe97a1f811f9be087,917986161bde1ada
COVID-2165.png,COVID,COVID/images/COVID-2165.png,COVID/masks/COVID-2165.png,True,G01269,test,a1ee49a3a8c79045ddf8fa63a937d9b8,d13f461a1e6f1c58
COVID-2166.png,COVID,COVID/images/COVID-2166.png,COVID/masks/COVID-2166.png,True,G01270,train,24d947351c14867a3c0239c4dbf4ada9,8d04430f4c6ffcea
COVID-2167.png,COVID,COVID/images/COVID-2167.png,COVID/masks/COVID-2167.png,True,G01271,train,850ed2c2417cb1aa31c88613fc6dd166,cd3c520b554e1f4d
COVID-2168.png,COVID,COVID/images/COVID-2168.png,COVID/masks/COVID-2168.png,True,G01272,train,95bf51c5c1dc22a7f81ef1e62593d734,917f061e1e5f191c
COVID-2169.png,COVID,COVID/images/COVID-2169.png,COVID/masks/COVID-2169.png,True,G01273,train,973c61134051ff2f949c9a1fcfb268cc,817b7f84141f035f
COVID-217.png,COVID,COVID/images/COVID-217.png,COVID/masks/COVID-217.png,True,G01274,train,e65ffbb682208dfd8491eb68cdc4ed97,d42e57c34d5d5c50
COVID-2170.png,COVID,COVID/images/COVID-2170.png,COVID/masks/COVID-2170.png,True,G01275,train,8d24b03ef39d8b60aaa88cfd40c033a3,917f06382d272d67
COVID-2171.png,COVID,COVID/images/COVID-2171.png,COVID/masks/COVID-2171.png,True,G01276,train,6818d9f5fac79c074f5f1398fc618541,d47d435f4f083c4a
COVID-2172.png,COVID,COVID/images/COVID-2172.png,COVID/masks/COVID-2172.png,True,G01277,train,65db122babe88e10188d09344606594c,d47ec3594d453a49
COVID-2173.png,COVID,COVID/images/COVID-2173.png,COVID/masks/COVID-2173.png,True,G01278,train,cda3e146f6c4b8aed2431c16af5b3315,9378ddf2c0c086af
COVID-2174.png,COVID,COVID/images/COVID-2174.png,COVID/masks/COVID-2174.png,True,G01279,train,2cf73e894d062ab75a6e3ec2dfd29a49,993d87461a4d455f
COVID-2175.png,COVID,COVID/images/COVID-2175.png,COVID/masks/COVID-2175.png,True,G01280,val,ae7996f62d10fc50f43c446a89af65e5,d52ed04b0be5461f
COVID-2176.png,COVID,COVID/images/COVID-2176.png,COVID/masks/COVID-2176.png,True,G01281,val,faff2e70a7243de8ffa3f906f409c41c,d939c61e5c415d59
COVID-2177.png,COVID,COVID/images/COVID-2177.png,COVID/masks/COVID-2177.png,True,G01282,test,f138f44335445dc55c3416c47fafff9d,817f861054d75b6d
COVID-2178.png,COVID,COVID/images/COVID-2178.png,COVID/masks/COVID-2178.png,True,G01283,test,ee9fd93a66b0a4f67f2175f35ebe164e,907b841c5a5f795a
COVID-2179.png,COVID,COVID/images/COVID-2179.png,COVID/masks/COVID-2179.png,True,G01284,test,1739409ee08e2bc9df73e6e3de49a738,9168c4547ecf7a51
COVID-218.png,COVID,COVID/images/COVID-218.png,COVID/masks/COVID-218.png,True,G01285,train,b648162a043140c8095e8ebd8a176d8d,953e1e1a5b487ae8
COVID-2180.png,COVID,COVID/images/COVID-2180.png,COVID/masks/COVID-2180.png,True,G01286,val,6efe5a57aa1118c9dc46f9fa1eef0dd6,947dc30b5f8b588a
COVID-2181.png,COVID,COVID/images/COVID-2181.png,COVID/masks/COVID-2181.png,True,G01287,train,2996629915fbcf2d824d8e95d03611ec,dd2c435b544b1d4e
COVID-2182.png,COVID,COVID/images/COVID-2182.png,COVID/masks/COVID-2182.png,True,G01288,train,2f4a0f838b3a98c2723cd72f6782244c,cd2dd05297461b5d
COVID-2183.png,COVID,COVID/images/COVID-2183.png,COVID/masks/COVID-2183.png,True,G01289,train,b92fa40e36bed2025f2dbfe6a6391052,d4f54b08d58b5b2a
COVID-2184.png,COVID,COVID/images/COVID-2184.png,COVID/masks/COVID-2184.png,True,G01290,train,022737055892381b0a8411f7969fbaff,d43d538ad18be649
COVID-2185.png,COVID,COVID/images/COVID-2185.png,COVID/masks/COVID-2185.png,True,G01291,train,919817723ab8acb84a0748382f7fa0da,d42c50d5135fc557
COVID-2186.png,COVID,COVID/images/COVID-2186.png,COVID/masks/COVID-2186.png,True,G01292,train,c5a7f3155e32f3a4f133bae0dd5af274,887f431a4f285f4d
COVID-2187.png,COVID,COVID/images/COVID-2187.png,COVID/masks/COVID-2187.png,True,G01293,train,fab4a785040baf446e7763a57ea23250,d03d465a1b6d3a3c
COVID-2188.png,COVID,COVID/images/COVID-2188.png,COVID/masks/COVID-2188.png,True,G01294,train,2a400daecdaa34d64444eeb7b4da3ec0,917b0a5e393931b3
COVID-2189.png,COVID,COVID/images/COVID-2189.png,COVID/masks/COVID-2189.png,True,G01295,train,63dde22b9fd3aae25b2d484c862ff5d7,9179c7c616105e5f
COVID-219.png,COVID,COVID/images/COVID-219.png,COVID/masks/COVID-219.png,True,G01296,train,5776866e49cfa37f7add5b53349a2f15,c87d424b1d5b4d1b
COVID-2190.png,COVID,COVID/images/COVID-2190.png,COVID/masks/COVID-2190.png,True,G01297,train,548795fde836397bafd86acd60e0d3c3,9979c7861a584f1d
COVID-2191.png,COVID,COVID/images/COVID-2191.png,COVID/masks/COVID-2191.png,True,G01298,val,5a22cf286eb23b3e251fb8246f358271,d52e2af1c855a956
COVID-2192.png,COVID,COVID/images/COVID-2192.png,COVID/masks/COVID-2192.png,True,G01299,test,802cd512533e9f6de805642af15962a6,817bc6461a386b7d
COVID-2193.png,COVID,COVID/images/COVID-2193.png,COVID/masks/COVID-2193.png,True,G01300,train,3cde2338216cd30b5206e0c4d51931b3,8360561e545f4f76
COVID-2194.png,COVID,COVID/images/COVID-2194.png,COVID/masks/COVID-2194.png,True,G01301,train,fb3f48682395e6551688ce1d4c0f0372,dd2cc34b154e1d4e
COVID-2195.png,COVID,COVID/images/COVID-2195.png,COVID/masks/COVID-2195.png,True,G01302,train,bb6a28825e913cb588108058f19a39ba,81795716965a1f5a
COVID-2196.png,COVID,COVID/images/COVID-2196.png,COVID/masks/COVID-2196.png,True,G01303,val,5097473c2be88f459448c35bd56aa08f,c13f86da14c53a9d
COVID-2197.png,COVID,COVID/images/COVID-2197.png,COVID/masks/COVID-2197.png,True,G01304,test,54405f075ee06a9e1479bbdfa65fc6aa,d42dd35d1e451d4c
COVID-2198.png,COVID,COVID/images/COVID-2198.png,COVID/masks/COVID-2198.png,True,G01305,test,2ae0599dbe67942d6992b56b00aac300,99fb861e8c4f190d
COVID-2199.png,COVID,COVID/images/COVID-2199.png,COVID/masks/COVID-2199.png,True,G01306,train,3f7b681ff0f2fa9a37658e01369c51dd,d47d430ac34f3c4b
COVID-22.png,COVID,COVID/images/COVID-22.png,COVID/masks/COVID-22.png,True,G01307,train,e0853b82e17fb29d08611d7671dd9132,c03d0f59186e79f4
COVID-220.png,COVID,COVID/images/COVID-220.png,COVID/masks/COVID-220.png,True,G01308,train,0502655828fb24e135d5c6eb6670c823,c80d871e47175bcb
COVID-2200.png,COVID,COVID/images/COVID-2200.png,COVID/masks/COVID-2200.png,True,G01309,train,3beaab4526499a94cd5cb22531d14ee3,81781e1a5f5f3a58
COVID-2201.png,COVID,COVID/images/COVID-2201.png,COVID/masks/COVID-2201.png,True,G01310,train,0942c524d606054cf9e5a4239dc57c91,81f816124f5e3b5e
COVID-2202.png,COVID,COVID/images/COVID-2202.png,COVID/masks/COVID-2202.png,True,G01311,train,9845ff37c0a3c14ad6ebe28a89c43877,b96f861849213bcf
COVID-2203.png,COVID,COVID/images/COVID-2203.png,COVID/masks/COVID-2203.png,True,G01312,train,3057a19ff98a8982a94418a92ac36af3,917f14185f1a4b3b
COVID-2204.png,COVID,COVID/images/COVID-2204.png,COVID/masks/COVID-2204.png,True,G01313,train,6105d9934ec12ea999a029ad225aed57,9c22436a697c7cf8
COVID-2205.png,COVID,COVID/images/COVID-2205.png,COVID/masks/COVID-2205.png,True,G01314,train,27982163eda878db52046222a613ca31,943d57511b5e551a
COVID-2206.png,COVID,COVID/images/COVID-2206.png,COVID/masks/COVID-2206.png,True,G01315,train,84c5cb2102db553438f95173eec5d9b5,847d43ca0f4ab54f
COVID-2207.png,COVID,COVID/images/COVID-2207.png,COVID/masks/COVID-2207.png,True,G01316,train,d90a1c8217de7fe4dc5d337250fd43a9,8178175a79787ba4
COVID-2208.png,COVID,COVID/images/COVID-2208.png,COVID/masks/COVID-2208.png,True,G01317,train,044b0a5c52b2101d792e6cab48ad1a8c,857c0f851d5d6c8d
COVID-2209.png,COVID,COVID/images/COVID-2209.png,COVID/masks/COVID-2209.png,True,G01318,val,53ac15f8996307f46bfcaab1cb54e800,b92d06965ad97878
COVID-221.png,COVID,COVID/images/COVID-221.png,COVID/masks/COVID-221.png,True,G01319,val,412b5600ccdcf260a307570d3b057887,843e075d7d6c6ce0
COVID-2210.png,COVID,COVID/images/COVID-2210.png,COVID/masks/COVID-2210.png,True,G01320,val,aa6fa16d2d13da8a44ba9034661f0375,95e9de1617051b1b
COVID-2211.png,COVID,COVID/images/COVID-2211.png,COVID/masks/COVID-2211.png,True,G01321,train,7a42eed286ed97450c46f35c689a9935,dd0cd4521f4f524b
COVID-2212.png,COVID,COVID/images/COVID-2212.png,COVID/masks/COVID-2212.png,True,G01322,test,f3007b99087a2f4781a0e9bfabf0d242,90bc4f5352474f1b
COVID-2213.png,COVID,COVID/images/COVID-2213.png,COVID/masks/COVID-2213.png,True,G01323,test,9c9e89c5537e3b4013f4e4e2c5e96ebf,813b47861f17471f
COVID-2214.png,COVID,COVID/images/COVID-2214.png,COVID/masks/COVID-2214.png,True,G01324,test,08203b2bb0e90140bd421920fa614856,992e061e1e4e7c7a
COVID-2215.png,COVID,COVID/images/COVID-2215.png,COVID/masks/COVID-2215.png,True,G01325,train,2dca8962368ff2d47a47ffa2f92f39a4,807947967bc16fb0
COVID-2216.png,COVID,COVID/images/COVID-2216.png,COVID/masks/COVID-2216.png,True,G01326,train,38906bcfd34300df5e0bed31289397a1,933d500a1f7f382b
COVID-2217.png,COVID,COVID/images/COVID-2217.png,COVID/masks/COVID-2217.png,True,G01327,test,c10ccfe8f131adc6baa64d281e193705,c136fa69054e2e67
COVID-2218.png,COVID,COVID/images/COVID-2218.png,COVID/masks/COVID-2218.png,True,G01328,train,96b4304706a7f23efc7c7e62721ab7b9,817a5e9a7a877a84
COVID-2219.png,COVID,COVID/images/COVID-2219.png,COVID/masks/COVID-2219.png,True,G01329,train,ebb7e8e147484be1cbecdb62e7532e39,916806563a157f5f
COVID-222.png,COVID,COVID/images/COVID-222.png,COVID/masks/COVID-222.png,True,G01330,train,486768317bf7b5d80a5f5eef6ce8504f,d52a434f4f0e0e2f
COVID-2220.png,COVID,COVID/images/COVID-2220.png,COVID/masks/COVID-2220.png,True,G01331,train,2cfdfeba6ce23674f13691e7762f2ac4,913d068616967f97
COVID-2221.png,COVID,COVID/images/COVID-2221.png,COVID/masks/COVID-2221.png,True,G01332,train,5a623eb2c09713c2198b982f5e72b0c5,ae22910b931f5f1e
COVID-2222.png,COVID,COVID/images/COVID-2222.png,COVID/masks/COVID-2222.png,True,G01333,train,412b1bb5337d1c1df35a17f1fda4882e,8178c6115f57515f
COVID-2223.png,COVID,COVID/images/COVID-2223.png,COVID/masks/COVID-2223.png,True,G01334,train,f6f4af4dc1c6fa31340593dc4a0a19f5,8160569e5ede781b
COVID-2224.png,COVID,COVID/images/COVID-2224.png,COVID/masks/COVID-2224.png,True,G01335,train,6e5b682619f51192a2953ab2198ff58a,d42c538b0e3c3cbb
COVID-2225.png,COVID,COVID/images/COVID-2225.png,COVID/masks/COVID-2225.png,True,G01336,train,847518fa7f939a687e581e57399593a3,d164c550784f7b6c
COVID-2226.png,COVID,COVID/images/COVID-2226.png,COVID/masks/COVID-2226.png,True,G01337,val,66edb3855ebc878e16d7ee27040ecacc,d52bc25b1fc2681d
COVID-2227.png,COVID,COVID/images/COVID-2227.png,COVID/masks/COVID-2227.png,True,G01338,train,aa4aff897a3daa197d7f0ffb04be3c84,d52a430b1f4d1b5e
COVID-2228.png,COVID,COVID/images/COVID-2228.png,COVID/masks/COVID-2228.png,True,G01339,train,52280a374b571f91a566fa1501ddaf6e,c428f7c35bd612d2
COVID-2229.png,COVID,COVID/images/COVID-2229.png,COVID/masks/COVID-2229.png,True,G01340,train,937951df6817193289f07c4e82787b71,d91917161a595f1e
COVID-223.png,COVID,COVID/images/COVID-223.png,COVID/masks/COVID-223.png,True,G01341,val,8d3b380925cb3b7e0b98d0664e54a8f1,817b16161e5f195b
COVID-2230.png,COVID,COVID/images/COVID-2230.png,COVID/masks/COVID-2230.png,True,G01342,val,e6870c541921c4166ccba3f0a15addf4,c07f42575b555558
COVID-2231.png,COVID,COVID/images/COVID-2231.png,COVID/masks/COVID-2231.png,True,G01343,test,1792c9b5a79a16c92d6213e06f9d25c8,813a56d27ac57b89
COVID-2232.png,COVID,COVID/images/COVID-2232.png,COVID/masks/COVID-2232.png,True,G01344,test,2672c87c7fd26885840d8ec0305403f2,813bf6d614d45ed0
COVID-2233.png,COVID,COVID/images/COVID-2233.png,COVID/masks/COVID-2233.png,True,G01345,train,60eb4953bdfe1c24baabef98a7cf47dd,992c861b5f4f184f
COVID-2234.png,COVID,COVID/images/COVID-2234.png,COVID/masks/COVID-2234.png,True,G01346,train,aea779f9daaa9ef01e1d816fdce42594,9578060bd9075f1f
COVID-2235.png,COVID,COVID/images/COVID-2235.png,COVID/masks/COVID-2235.png,True,G01347,val,25799eed09f5e8eaeccd8ea7643236e6,dc2ad31e4e4f5849
COVID-2236.png,COVID,COVID/images/COVID-2236.png,COVID/masks/COVID-2236.png,True,G01348,train,60a8ed887ad91a75e123a910d7790369,d17dc64a1ac07bd8
COVID-2237.png,COVID,COVID/images/COVID-2237.png,COVID/masks/COVID-2237.png,True,G01349,val,1906dc0033ea5ae9eb370875d3230fe6,807f06165f595e5c
COVID-2238.png,COVID,COVID/images/COVID-2238.png,COVID/masks/COVID-2238.png,True,G01350,train,9842d0872bbe3d14bab39747ef1382ac,d42ec24b4f464f5c
COVID-2239.png,COVID,COVID/images/COVID-2239.png,COVID/masks/COVID-2239.png,True,G01351,train,59332dad7ab3f5beeab6248abf6cc1fe,d42cd2d10bd7431f
COVID-224.png,COVID,COVID/images/COVID-224.png,COVID/masks/COVID-224.png,True,G01352,test,6330d5f26f9f313b6f08d23f96a5638f,bd6dc61617451a1d
COVID-2240.png,COVID,COVID/images/COVID-2240.png,COVID/masks/COVID-2240.png,True,G01353,train,c43c7c6d5678cac27e91c31e33fd3d54,897906561a4f7e3a
COVID-2241.png,COVID,COVID/images/COVID-2241.png,COVID/masks/COVID-2241.png,True,G01354,train,c83cf5643d58d2e854df673e2901c267,d42fc35d164f1954
COVID-2242.png,COVID,COVID/images/COVID-2242.png,COVID/masks/COVID-2242.png,True,G01355,train,e74fbf19e1c756728fdddf7bfe55b5d9,d03b5a082b1d3f9d
COVID-2243.png,COVID,COVID/images/COVID-2243.png,COVID/masks/COVID-2243.png,True,G01356,train,d514ba4cd3e7176c0f3ba12600f3db35,c47c9b09060f7d2f
COVID-2244.png,COVID,COVID/images/COVID-2244.png,COVID/masks/COVID-2244.png,True,G01357,train,b31c7f03a9a66827387f8a58678b648f,d42f935189b3474d
COVID-2245.png,COVID,COVID/images/COVID-2245.png,COVID/masks/COVID-2245.png,True,G01358,train,b7082118204444e31c99d65a14922844,c13bc696144f9b1b
COVID-2246.png,COVID,COVID/images/COVID-2246.png,COVID/masks/COVID-2246.png,True,G01359,train,dbabb6d8805d7838f41a65ea6c9994af,d53f474a194f4519
COVID-2247.png,COVID,COVID/images/COVID-2247.png,COVID/masks/COVID-2247.png,True,G01360,train,7ad713921ea9e9ed3a44e1c15194173c,817d07061e1f1f5b
COVID-2248.png,COVID,COVID/images/COVID-2248.png,COVID/masks/COVID-2248.png,True,G01361,train,9c65dcff47156ade0c2fe9d0e02166b0,887917561e586b79
COVID-2249.png,COVID,COVID/images/COVID-2249.png,COVID/masks/COVID-2249.png,True,G01362,train,d74cb2d39b13c02ed208e82ca8267adc,dc4b410f0c2ebcbd
COVID-225.png,COVID,COVID/images/COVID-225.png,COVID/masks/COVID-225.png,True,G01363,val,4536b4b94cc06ac9819cfd6a407dddc1,c02e430a3c2fbdfc
COVID-2250.png,COVID,COVID/images/COVID-2250.png,COVID/masks/COVID-2250.png,True,G01364,test,3647e0457a57003b94e11d25c5a00c2c,8039c25593c7579f
COVID-2251.png,COVID,COVID/images/COVID-2251.png,COVID/masks/COVID-2251.png,True,G01365,test,68a7447a7d2fc08c39c2784d1f2d2949,dd2c430b454f5d4d
COVID-2252.png,COVID,COVID/images/COVID-2252.png,COVID/masks/COVID-2252.png,True,G01366,train,87e50c606fa3b666e5a4b364b66ff73a,d03d4b4c0a2f3d3e
COVID-2253.png,COVID,COVID/images/COVID-2253.png,COVID/masks/COVID-2253.png,True,G01367,train,1d181ce3b5b44569798fdd264a91882c,d07d42834b2b4f2f
COVID-2254.png,COVID,COVID/images/COVID-2254.png,COVID/masks/COVID-2254.png,True,G01368,test,98c6d647f7c86a506a81740eab7a7b34,d47c075b19025f1f
COVID-2255.png,COVID,COVID/images/COVID-2255.png,COVID/masks/COVID-2255.png,True,G01369,train,8c4e91eaf779f673e3e97925d0429b61,d42f435d0a6e0f4e
COVID-2256.png,COVID,COVID/images/COVID-2256.png,COVID/masks/COVID-2256.png,True,G01370,train,e678e5783327d8b7160166bba816eac9,817a121e5f3a5b0f
COVID-2257.png,COVID,COVID/images/COVID-2257.png,COVID/masks/COVID-2257.png,True,G01371,train,25d22006073e31463340bc550900c2f9,816cc6165b457b57
COVID-2258.png,COVID,COVID/images/COVID-2258.png,COVID/masks/COVID-2258.png,True,G01372,train,c2fad859a5a8ae142f4f3b03cf82011e,d42bd34f4868392f
COVID-2259.png,COVID,COVID/images/COVID-2259.png,COVID/masks/COVID-2259.png,True,G01373,train,f2bbaff48741c1a4a0774689479ba43a,d42f53434e6a683d
COVID-226.png,COVID,COVID/images/COVID-226.png,COVID/masks/COVID-226.png,True,G01374,train,e2815f2b4fe712a9994d2e62bb43f91f,d939c7dc19ce5818
COVID-2260.png,COVID,COVID/images/COVID-2260.png,COVID/masks/COVID-2260.png,True,G01375,train,18bcecada311deddefb9648453a2440c,dc7c42c30fc22b9d
COVID-2261.png,COVID,COVID/images/COVID-2261.png,COVID/masks/COVID-2261.png,True,G01376,train,c91446e643887e4f48a999447842d80a,d02bd2504c6a797f
COVID-2262.png,COVID,COVID/images/COVID-2262.png,COVID/masks/COVID-2262.png,True,G01377,train,aaad813921120290e74ff57ee466016b,c47d42451f427f1b
COVID-2263.png,COVID,COVID/images/COVID-2263.png,COVID/masks/COVID-2263.png,True,G01378,train,f91006bf7475da109d38bda6f75de200,846e07190d4f5e5f
COVID-2264.png,COVID,COVID/images/COVID-2264.png,COVID/masks/COVID-2264.png,True,G01379,train,40030f7818f015d5fed0ca6b8a4b0fc4,d42e42d7194c5f59
COVID-2265.png,COVID,COVID/images/COVID-2265.png,COVID/masks/COVID-2265.png,True,G01380,train,64265f5c798842b2fd8620af19f1004a,817fc6861955cd55
COVID-2266.png,COVID,COVID/images/COVID-2266.png,COVID/masks/COVID-2266.png,True,G01381,val,7a58ee80cd2fec0cac6702dea2eafc89,946e475c6c78f8e0
COVID-2267.png,COVID,COVID/images/COVID-2267.png,COVID/masks/COVID-2267.png,True,G01382,val,f687ac6457e5830818b3a294559ac141,817906161e5e5f1f
COVID-2268.png,COVID,COVID/images/COVID-2268.png,COVID/masks/COVID-2268.png,True,G01383,val,8cab33441488ccd7b4863eb1a9aa5a0b,8178061e5e5f7a5a
COVID-2269.png,COVID,COVID/images/COVID-2269.png,COVID/masks/COVID-2269.png,True,G01384,test,317bd05a8b3ec09802bed302b8b5e234,d94ed248e2bae6b0
COVID-227.png,COVID,COVID/images/COVID-227.png,COVID/masks/COVID-227.png,True,G01385,test,c9b4113ceaad3c125b8a64467a41abda,8178141e7b3b2b3b
COVID-2270.png,COVID,COVID/images/COVID-2270.png,COVID/masks/COVID-2270.png,True,G01386,train,3c48661f27e5373d39f9130c7e9bb2e7,d43dc255dd42455b
COVID-2271.png,COVID,COVID/images/COVID-2271.png,COVID/masks/COVID-2271.png,True,G01387,train,f21b9e20a3b296b934af21c4b3f4e006,dc2c821f9c4f3c78
COVID-2272.png,COVID,COVID/images/COVID-2272.png,COVID/masks/COVID-2272.png,True,G01388,train,c64f207b4c6086e4e19bdb4c1ec4da48,9169061e4bd979cd
COVID-2273.png,COVID,COVID/images/COVID-2273.png,COVID/masks/COVID-2273.png,True,G01389,train,cb54025c7bdf259648135e4692ea6249,dc2ec3131bc3471d
COVID-2274.png,COVID,COVID/images/COVID-2274.png,COVID/masks/COVID-2274.png,True,G01390,train,e6ae98e39a05df5622dd391a78086be3,d138475f584f191b
COVID-2275.png,COVID,COVID/images/COVID-2275.png,COVID/masks/COVID-2275.png,True,G01391,train,cd5e9e7e416e0fc449af3514b2225ea2,8879d61ec85b7949
COVID-2276.png,COVID,COVID/images/COVID-2276.png,COVID/masks/COVID-2276.png,True,G01392,train,0efd731fb24a1fa307e14e52f75b5bff,9528d6d71ad4471b
COVID-2277.png,COVID,COVID/images/COVID-2277.png,COVID/masks/COVID-2277.png,True,G01393,val,3a5bc9b340d67e9676c90533d5e658b9,d16bc61f18455d59
COVID-2278.png,COVID,COVID/images/COVID-2278.png,COVID/masks/COVID-2278.png,True,G01394,train,a6357c7a186164a1a5a82f9643541beb,c635d1d14dd8d14d
COVID-2279.png,COVID,COVID/images/COVID-2279.png,COVID/masks/COVID-2279.png,True,G01395,train,329b3f00ab62fbcec2e941743e51f6c9,8839d2659fd4d21d
COVID-228.png,COVID,COVID/images/COVID-228.png,COVID/masks/COVID-228.png,True,G01396,train,ea4807d54e06e80bf094a186a0624ebd,d86c471a1c2e3dbc
COVID-2280.png,COVID,COVID/images/COVID-2280.png,COVID/masks/COVID-2280.png,True,G01397,train,f6fe387fc02dbffb6bcb287864d4cc77,997f071e184f1b1c
COVID-2281.png,COVID,COVID/images/COVID-2281.png,COVID/masks/COVID-2281.png,True,G01398,val,46eac6bc1858fadcb85279797e927d99,d467c35b4d7c6064
COVID-2282.png,COVID,COVID/images/COVID-2282.png,COVID/masks/COVID-2282.png,True,G01399,train,332a09d908b7e4932a8a01ec23bdaa37,f969c65e5841391b
COVID-2283.png,COVID,COVID/images/COVID-2283.png,COVID/masks/COVID-2283.png,True,G01400,test,7ef4246a12772f70ac6140fd752dec08,dd2cd2575d4c405d
COVID-2284.png,COVID,COVID/images/COVID-2284.png,COVID/masks/COVID-2284.png,True,G01401,train,095b9ef8c5c70626f7f827d38cd414d5,996986961ac7a73a
COVID-2285.png,COVID,COVID/images/COVID-2285.png,COVID/masks/COVID-2285.png,True,G01402,val,9c5ae5e472f3ffe805268d60fba08004,d43dc2531fc34b0b
COVID-2286.png,COVID,COVID/images/COVID-2286.png,COVID/masks/COVID-2286.png,True,G01403,train,60481c65b66a82a6a9ef05ab2d7db20b,dc6d13194e394d5c
COVID-2287.png,COVID,COVID/images/COVID-2287.png,COVID/masks/COVID-2287.png,True,G01404,test,4949b9a01230a6f33124df469112391e,c875cb5bc3491ac9
COVID-2288.png,COVID,COVID/images/COVID-2288.png,COVID/masks/COVID-2288.png,True,G01405,test,ce6f7021873fe7d29ad89772945fc3ff,996c1e5a4b794879
COVID-2289.png,COVID,COVID/images/COVID-2289.png,COVID/masks/COVID-2289.png,True,G01406,train,096409b2669eeaf210b61ddf57680396,d43843030f1b3f3f
COVID-229.png,COVID,COVID/images/COVID-229.png,COVID/masks/COVID-229.png,True,G00759,val,99c7141f3d994cf7431bff31262bb916,d82fc61e8ccc1dc6
COVID-2290.png,COVID,COVID/images/COVID-2290.png,COVID/masks/COVID-2290.png,True,G01407,train,ac89ca86eeb2dd2d94637aa5cacf2c97,883fd75b4d4b184c
COVID-2291.png,COVID,COVID/images/COVID-2291.png,COVID/masks/COVID-2291.png,True,G01408,train,99b41cc0fa18d487be70b449f6d111ac,d53ad25b55424b55
COVID-2292.png,COVID,COVID/images/COVID-2292.png,COVID/masks/COVID-2292.png,True,G01409,train,f228cc19b7f201f81e65536fe8a76a8d,d1398616181f573f
COVID-2293.png,COVID,COVID/images/COVID-2293.png,COVID/masks/COVID-2293.png,True,G01410,train,7ecc535a2788ac9fd00b9437f8be5dbd,dc60179b2a986bbc
COVID-2294.png,COVID,COVID/images/COVID-2294.png,COVID/masks/COVID-2294.png,True,G01411,train,3649f78065d944bccd37a2d3a772506f,d52ec2971bc40f1e
COVID-2295.png,COVID,COVID/images/COVID-2295.png,COVID/masks/COVID-2295.png,True,G01412,train,ddb873324536c796561061d077e0e8c6,d42d871b59d51e38
COVID-2296.png,COVID,COVID/images/COVID-2296.png,COVID/masks/COVID-2296.png,True,G01413,train,a0f1ebd0a22317e3c75a84af0a4a067b,cc1a430f2e2db735
COVID-2297.png,COVID,COVID/images/COVID-2297.png,COVID/masks/COVID-2297.png,True,G01414,train,0e72246f0a183c3cc58d7250aaa0eb82,842e831bd91dfa4b
COVID-2298.png,COVID,COVID/images/COVID-2298.png,COVID/masks/COVID-2298.png,True,G01415,train,c3a2e5faeefb8674a511a67600625c65,cc3d4b0c2e2c35b7
COVID-2299.png,COVID,COVID/images/COVID-2299.png,COVID/masks/COVID-2299.png,True,G01416,train,b6f1154fefd6b6128dd42a3208c74189,93796c9c788764a6
COVID-23.png,COVID,COVID/images/COVID-23.png,COVID/masks/COVID-23.png,True,G01417,train,e939ac50b30eb8f6404543f09fee252a,916bc796c6c6d612
COVID-230.png,COVID,COVID/images/COVID-230.png,COVID/masks/COVID-230.png,True,G00288,train,503942315f3ebffa3b1a456a40326557,883f831b290d1f3f
COVID-2300.png,COVID,COVID/images/COVID-2300.png,COVID/masks/COVID-2300.png,True,G01418,val,ff39a59d9f6f229d674aeaba332fc821,d42c53d35f4eaa50
COVID-2301.png,COVID,COVID/images/COVID-2301.png,COVID/masks/COVID-2301.png,True,G01419,val,503e985778704d197f0cd23e515f2c2b,d87b53082c2b3d2f
COVID-2302.png,COVID,COVID/images/COVID-2302.png,COVID/masks/COVID-2302.png,True,G01420,val,5538c777cf3dae9143ffe45bd664313c,c42f430b2e2f2d2f
COVID-2303.png,COVID,COVID/images/COVID-2303.png,COVID/masks/COVID-2303.png,True,G01421,train,4566fc19b236303bea3a72335c4740b1,817b06161a5f1d5f
COVID-2304.png,COVID,COVID/images/COVID-2304.png,COVID/masks/COVID-2304.png,True,G01422,train,c735890d999c57a332dd1075660d8305,997f461a595b184b
COVID-2305.png,COVID,COVID/images/COVID-2305.png,COVID/masks/COVID-2305.png,True,G01423,test,1c4f21f7fbc43137aa09878dbb44b882,d4785b4b5b496c2c
COVID-2306.png,COVID,COVID/images/COVID-2306.png,COVID/masks/COVID-2306.png,True,G01424,test,9a80efb694706e3f03c2ca01d71045b6,dd2a43031f1d165f
COVID-2307.png,COVID,COVID/images/COVID-2307.png,COVID/masks/COVID-2307.png,True,G01425,test,a19ceb27160984f0ad63f19a0a0f7c90,d03b470e1f4f1f18
COVID-2308.png,COVID,COVID/images/COVID-2308.png,COVID/masks/COVID-2308.png,True,G01426,train,7dece96de805760493e1cca005df99ec,917f861a3e4d3938
COVID-2309.png,COVID,COVID/images/COVID-2309.png,COVID/masks/COVID-2309.png,True,G01427,train,7e2681a9cc24b872d6f9b3ec6921c0f3,dc2a530f4e1a6c6e
COVID-231.png,COVID,COVID/images/COVID-231.png,COVID/masks/COVID-231.png,True,G01428,train,366ad0d9be07a27c02de2345ed9c4e80,817906865e1e5e5f
COVID-2310.png,COVID,COVID/images/COVID-2310.png,COVID/masks/COVID-2310.png,True,G01429,train,0c1215ac37c68f41a034208609d97dbf,d97d167a2d584958
COVID-2311.png,COVID,COVID/images/COVID-2311.png,COVID/masks/COVID-2311.png,True,G01430,train,2f7eea45c3a808d3b10c0b02ffc9dc49,943c43561a5b5b1f
COVID-2312.png,COVID,COVID/images/COVID-2312.png,COVID/masks/COVID-2312.png,True,G01431,train,a73e7cbb6df1e3d061ba276a3feea98a,d13b52515a0f5b1b
COVID-2313.png,COVID,COVID/images/COVID-2313.png,COVID/masks/COVID-2313.png,True,G01432,train,9e75c3e413336ce70a222545e4ba22f9,dd0c461b1d4f185f
COVID-2314.png,COVID,COVID/images/COVID-2314.png,COVID/masks/COVID-2314.png,True,G01433,train,c23925e0d7e9e007b72a3804f1f94853,9d0d46165e4e5a79
COVID-2315.png,COVID,COVID/images/COVID-2315.png,COVID/masks/COVID-2315.png,True,G01434,train,6b2a7e92cdf9248b8b56ad2bf7fe6a9b,913dc61a584f5b1b
COVID-2316.png,COVID,COVID/images/COVID-2316.png,COVID/masks/COVID-2316.png,True,G01435,train,566ee7718d6ca5b2100daea15e012953,997b861e185f1b19
COVID-2317.png,COVID,COVID/images/COVID-2317.png,COVID/masks/COVID-2317.png,True,G01436,val,1c7bdec38da91394ed7b07df9a8c4578,817f041a5f5b135b
COVID-2318.png,COVID,COVID/images/COVID-2318.png,COVID/masks/COVID-2318.png,True,G01437,train,734bfe72095f1e7f335ccd96f4bb7c5b,8c4f434d6c7ce8b8
COVID-2319.png,COVID,COVID/images/COVID-2319.png,COVID/masks/COVID-2319.png,True,G01438,val,20e4a7ff34a22f8d1d600762e6e06990,997d465a144f3959
COVID-232.png,COVID,COVID/images/COVID-232.png,COVID/masks/COVID-232.png,True,G01439,val,dd03874ebbbbb6edaf1ca7156607c410,dd2cc21bc44f1d47
COVID-2320.png,COVID,COVID/images/COVID-2320.png,COVID/masks/COVID-2320.png,True,G01440,train,0bc6c7b49c96e83308e69133ee9add5a,985f061e58491f7e
COVID-2321.png,COVID,COVID/images/COVID-2321.png,COVID/masks/COVID-2321.png,True,G01441,train,3b5c9d55818e605499863eb650cef7b9,81798497169ace5f
COVID-2322.png,COVID,COVID/images/COVID-2322.png,COVID/masks/COVID-2322.png,True,G01442,train,64e76101db5e3dc6c3b2d197625c5151,987d86c7121ac7c7
COVID-2323.png,COVID,COVID/images/COVID-2323.png,COVID/masks/COVID-2323.png,True,G01443,val,e96c67600884bba6a79c84641385397d,d925c61a465b1b4f
COVID-2324.png,COVID,COVID/images/COVID-2324.png,COVID/masks/COVID-2324.png,True,G01444,train,754c29cf3f2a4c8f0b50714efb639d4b,817bc516134f7b51
COVID-2325.png,COVID,COVID/images/COVID-2325.png,COVID/masks/COVID-2325.png,True,G01445,val,7918cfb1c9f251140bfacefc6ddc4cd1,d03b839e0cdb98c7
COVID-2326.png,COVID,COVID/images/COVID-2326.png,COVID/masks/COVID-2326.png,True,G01446,val,281fe5ee9d511bc939bb727980504dbc,9928061f1e4f5f33
COVID-2327.png,COVID,COVID/images/COVID-2327.png,COVID/masks/COVID-2327.png,True,G01447,test,31fcb2b2a9cfc68a11ee358462258b07,d83985d71e5a491d
COVID-2328.png,COVID,COVID/images/COVID-2328.png,COVID/masks/COVID-2328.png,True,G01448,test,e00a3c2a221f9a055c2b1c1df4bc83e3,98794386be716965
COVID-2329.png,COVID,COVID/images/COVID-2329.png,COVID/masks/COVID-2329.png,True,G01449,test,66ecb8be7321f326438eec6910276059,8c7d82131b495f3e
COVID-233.png,COVID,COVID/images/COVID-233.png,COVID/masks/COVID-233.png,True,G01450,train,4f9cd146fb7546641637a893ce2dfa97,d42e4b0b3cf531d2
COVID-2330.png,COVID,COVID/images/COVID-2330.png,COVID/masks/COVID-2330.png,True,G01451,train,2188be5a88f1ae27f998562e52a0e6b0,c42f530b2c2e3e9e
COVID-2331.png,COVID,COVID/images/COVID-2331.png,COVID/masks/COVID-2331.png,True,G01452,train,fe54eb642e78952b4b7908b7cdce277a,9378165b7a397984
COVID-2332.png,COVID,COVID/images/COVID-2332.png,COVID/masks/COVID-2332.png,True,G01453,train,56a4010bcfbd1b55df5f5b56cb97a1ae,c03fc2515ecf5919
COVID-2333.png,COVID,COVID/images/COVID-2333.png,COVID/masks/COVID-2333.png,True,G01454,train,bbd4f7d7bcbf10fd8b14718c95eb761c,997d870612875f17
COVID-2334.png,COVID,COVID/images/COVID-2334.png,COVID/masks/COVID-2334.png,True,G01455,train,f185e51450adb095b23f9ac19bb28a80,d46d534b4f484d4b
COVID-2335.png,COVID,COVID/images/COVID-2335.png,COVID/masks/COVID-2335.png,True,G01456,train,800da262652a7b4350219244cc6d531c,993a931b450f195f
COVID-2336.png,COVID,COVID/images/COVID-2336.png,COVID/masks/COVID-2336.png,True,G01457,train,135655148745055777aef6cc1b2059c9,8c57134d0f3c6c6e
COVID-2337.png,COVID,COVID/images/COVID-2337.png,COVID/masks/COVID-2337.png,True,G01458,train,68a8160a7116604f48974885ea3f7be7,817b431a1f4f0b3b
COVID-2338.png,COVID,COVID/images/COVID-2338.png,COVID/masks/COVID-2338.png,True,G01459,test,4ffca159599dcbfbc3a1749ff474e78e,8179061a5f785b7a
COVID-2339.png,COVID,COVID/images/COVID-2339.png,COVID/masks/COVID-2339.png,True,G01460,train,0f6e98838c88669c02c330de593731f9,cc0c470b5d4d5f4d
COVID-234.png,COVID,COVID/images/COVID-234.png,COVID/masks/COVID-234.png,True,G01461,train,1d901060f57b6c25f6d2bc2106d77eb7,9d2cc21b445f7c6c
COVID-2340.png,COVID,COVID/images/COVID-2340.png,COVID/masks/COVID-2340.png,True,G01462,train,cc29bb052eb8ff5c20890c7b94cb2c96,9d2cc25b0b4c3e79
COVID-2341.png,COVID,COVID/images/COVID-2341.png,COVID/masks/COVID-2341.png,True,G01463,test,24db2fd15dbb2fe774c64aa36c6274da,d42ac34f0ecc2dc7
COVID-2342.png,COVID,COVID/images/COVID-2342.png,COVID/masks/COVID-2342.png,True,G01464,train,637622befa40f4658949d7e30874288d,847f22932f857f81
COVID-2343.png,COVID,COVID/images/COVID-2343.png,COVID/masks/COVID-2343.png,True,G01465,test,7c448ad9fd92bcdabdc17019ad1ab65e,817046da5ad64fd6
COVID-2344.png,COVID,COVID/images/COVID-2344.png,COVID/masks/COVID-2344.png,True,G01466,train,4863c584842d5f53fd7b375a100c8e7d,81781e1a195d7b5d
COVID-2345.png,COVID,COVID/images/COVID-2345.png,COVID/masks/COVID-2345.png,True,G01467,train,49326fa16dd41021c1a45ed8cbae6f82,812947d61e67c717
COVID-2346.png,COVID,COVID/images/COVID-2346.png,COVID/masks/COVID-2346.png,True,G01468,train,2ac60b798f5dad42334dee8e59f18c3b,83600fcf15b50f5e
COVID-2347.png,COVID,COVID/images/COVID-2347.png,COVID/masks/COVID-2347.png,True,G01469,train,8bfa66c6bfdd8ca1e8e6756c4368e577,8179c6141e5f5a1f
COVID-2348.png,COVID,COVID/images/COVID-2348.png,COVID/masks/COVID-2348.png,True,G01470,train,b7a2bcc72cef767563d9afab2041cbdd,d6e721c02bc22fce
COVID-2349.png,COVID,COVID/images/COVID-2349.png,COVID/masks/COVID-2349.png,True,G01471,train,d3af0e205c1f6b6020a40ed87d032d1e,d62f50ca896e4b0f
COVID-235.png,COVID,COVID/images/COVID-235.png,COVID/masks/COVID-235.png,True,G01472,train,72c150a799a99b3f1dd9af334ad6b098,9179061e5f5a3c5a
COVID-2350.png,COVID,COVID/images/COVID-2350.png,COVID/masks/COVID-2350.png,True,G01473,train,adea5b7c7c8292adab214cee8f7051be,d4bc830b8b0f394f
COVID-2351.png,COVID,COVID/images/COVID-2351.png,COVID/masks/COVID-2351.png,True,G01474,train,26866ae8bbc872180d9e686134e2a0f9,89209e1e38df78ba
COVID-2352.png,COVID,COVID/images/COVID-2352.png,COVID/masks/COVID-2352.png,True,G01475,train,70ab9b1fb47165d2372a61ea0f47ddb0,9039d68e5e1b7d18
COVID-2353.png,COVID,COVID/images/COVID-2353.png,COVID/masks/COVID-2353.png,True,G01476,train,cbd7b0f7d2c8218793428eb45b21632f,d42c53530f0f4e5e
COVID-2354.png,COVID,COVID/images/COVID-2354.png,COVID/masks/COVID-2354.png,True,G01477,train,7f39cac9e522069d9409b45ab25f3d0a,dc54034b2f6c3d2d
COVID-2355.png,COVID,COVID/images/COVID-2355.png,COVID/masks/COVID-2355.png,True,G01478,train,ea39be72610db5573f257be8ed428ef9,9138061a1f7f79e1
COVID-2356.png,COVID,COVID/images/COVID-2356.png,COVID/masks/COVID-2356.png,True,G01479,train,3f99b9a8f974bd778e925a0aebfc59aa,d52906d49a5aed17
COVID-2357.png,COVID,COVID/images/COVID-2357.png,COVID/masks/COVID-2357.png,True,G01480,test,3de036d165cfd557c26f9a8a8a3bf019,91788c1e1a5e7a79
COVID-2358.png,COVID,COVID/images/COVID-2358.png,COVID/masks/COVID-2358.png,True,G01481,val,87ad7df164fd7c3df770bc2d7e2d226f,837e543e686b6165
COVID-2359.png,COVID,COVID/images/COVID-2359.png,COVID/masks/COVID-2359.png,True,G01482,val,3cb1131b122380ad6b28e0c7e02b66d4,c83a971f1e1cb15a
COVID-236.png,COVID,COVID/images/COVID-236.png,COVID/masks/COVID-236.png,True,G01483,val,5744352bc4e916dcc0238182a7b1e19a,8168065e1e5e7b79
COVID-2360.png,COVID,COVID/images/COVID-2360.png,COVID/masks/COVID-2360.png,True,G01484,test,184508fb4bb972f79222f59e82b320db,827f5c1d7062626f
COVID-2361.png,COVID,COVID/images/COVID-2361.png,COVID/masks/COVID-2361.png,True,G01485,train,4743cbad2b8621d2dafc333ef88520d7,917b440c1c4f7e3d
COVID-2362.png,COVID,COVID/images/COVID-2362.png,COVID/masks/COVID-2362.png,True,G01486,test,327c8ca3a8acf4e8b5042c7fcba3727c,be2e674383cbc309
COVID-2363.png,COVID,COVID/images/COVID-2363.png,COVID/masks/COVID-2363.png,True,G01487,train,cef14db08d5769773238105850f1e7e9,946c465a19cfcb59
COVID-2364.png,COVID,COVID/images/COVID-2364.png,COVID/masks/COVID-2364.png,True,G01488,train,479037194454b5a4db2ff9d02182c527,994f869be9fca0a0
COVID-2365.png,COVID,COVID/images/COVID-2365.png,COVID/masks/COVID-2365.png,True,G01489,train,b95726d12d47e074de8eee601b9030df,dc7f875a4b165a40
COVID-2366.png,COVID,COVID/images/COVID-2366.png,COVID/masks/COVID-2366.png,True,G01490,train,07134df437b42fc02b5842aa9a043a7e,d42d431a2fc33bc5
COVID-2367.png,COVID,COVID/images/COVID-2367.png,COVID/masks/COVID-2367.png,True,G01491,train,8cf4b5fd69430ad7b69b58f4263a291d,817b161a5f5e3869
COVID-2368.png,COVID,COVID/images/COVID-2368.png,COVID/masks/COVID-2368.png,True,G01492,train,6d2ec2bb1ff46dc83e7c9a43592da507,b87dc5875a82169e
COVID-2369.png,COVID,COVID/images/COVID-2369.png,COVID/masks/COVID-2369.png,True,G01493,test,90ce4f99213e65945f87258f5d81aa3b,dc28d34f5f445c49
COVID-237.png,COVID,COVID/images/COVID-237.png,COVID/masks/COVID-237.png,True,G01494,train,ff64d0a766c4ddbd005b63c6fc17ce28,9d2dc3131f461b1e
COVID-2370.png,COVID,COVID/images/COVID-2370.png,COVID/masks/COVID-2370.png,True,G01495,train,27fbdd90f08b9034ebe2a1aa93339c62,d91f16180f1a9dba
COVID-2371.png,COVID,COVID/images/COVID-2371.png,COVID/masks/COVID-2371.png,True,G01496,val,c00c26c3ecc90ce55a24a506fee5b144,dd2f425f174e0e18
COVID-2372.png,COVID,COVID/images/COVID-2372.png,COVID/masks/COVID-2372.png,True,G01497,test,2b8028dfb1c9d449525bb472b1172ef0,9c7e461a196d187d
COVID-2373.png,COVID,COVID/images/COVID-2373.png,COVID/masks/COVID-2373.png,True,G01498,train,430c50f36d15914432abd304ce2ce6e2,d52955021f3f5f06
COVID-2374.png,COVID,COVID/images/COVID-2374.png,COVID/masks/COVID-2374.png,True,G01499,train,e9645be750c8665459243cdf995136cc,c43cc75d57471985
COVID-2375.png,COVID,COVID/images/COVID-2375.png,COVID/masks/COVID-2375.png,True,G01500,train,396bc893f4aa2576e809937fe221f1fe,dd6ac21b494c1de9
COVID-2376.png,COVID,COVID/images/COVID-2376.png,COVID/masks/COVID-2376.png,True,G01501,train,fc23d52a8f6c979789b9ee36110daebb,d84e1e5d1e3d86b0
COVID-2377.png,COVID,COVID/images/COVID-2377.png,COVID/masks/COVID-2377.png,True,G01502,train,e04a02808833f025de523fad58b9a78e,81fac658597b594c
COVID-2378.png,COVID,COVID/images/COVID-2378.png,COVID/masks/COVID-2378.png,True,G01503,val,708fb7cd54067e35c7b8b34667d28847,d42e43574a4f4f0d
COVID-2379.png,COVID,COVID/images/COVID-2379.png,COVID/masks/COVID-2379.png,True,G01504,train,937a513b4f0d699a26f0600a69c2571f,815b1e16185b5b7b
COVID-238.png,COVID,COVID/images/COVID-238.png,COVID/masks/COVID-238.png,True,G01505,val,8351dbdf7d51c3863564f2cb7d2d1545,d46c060b1d7cece5
COVID-2380.png,COVID,COVID/images/COVID-2380.png,COVID/masks/COVID-2380.png,True,G01506,test,e35bebf5257e0d1cfb68062b017b2ae6,d92907861f1d0f1f
COVID-2381.png,COVID,COVID/images/COVID-2381.png,COVID/masks/COVID-2381.png,True,G01507,train,f4e3be87c799b50676b267fdfbe18afd,d43c0242c34fd75f
COVID-2382.png,COVID,COVID/images/COVID-2382.png,COVID/masks/COVID-2382.png,True,G01508,val,7804ef50e4dd37545f9d865020075412,dc4f4b0c0f2925b7
COVID-2383.png,COVID,COVID/images/COVID-2383.png,COVID/masks/COVID-2383.png,True,G01509,train,81b3dcf35137687cb1e35ea59eacd22f,812856d61f794f5a
COVID-2384.png,COVID,COVID/images/COVID-2384.png,COVID/masks/COVID-2384.png,True,G01510,train,406da3c5cec6e2ed4ba2db031105fbc7,d43d035f0a5f1a4e
COVID-2385.png,COVID,COVID/images/COVID-2385.png,COVID/masks/COVID-2385.png,True,G01511,train,24529157aee60fe2aed8658a08164dab,817f871a9acf5a50
COVID-2386.png,COVID,COVID/images/COVID-2386.png,COVID/masks/COVID-2386.png,True,G01512,train,6006a64a089a1cad30a581ad76ec1b2a,817b04161f1b5f1f
COVID-2387.png,COVID,COVID/images/COVID-2387.png,COVID/masks/COVID-2387.png,True,G01513,train,201fd3947d2538e70fc760dcfb9ffe14,d43d53434e5a5b4a
COVID-2388.png,COVID,COVID/images/COVID-2388.png,COVID/masks/COVID-2388.png,True,G01514,train,fd7305b1553a433ee47b4e24b82a28b0,8978869696c63577
COVID-2389.png,COVID,COVID/images/COVID-2389.png,COVID/masks/COVID-2389.png,True,G01515,train,e4e78cceb41a8fe3e9e7def802e39c7c,c12bc58e18bdcdcc
COVID-239.png,COVID,COVID/images/COVID-239.png,COVID/masks/COVID-239.png,True,G01516,train,9b4006c87456ae636291c812bc279731,a879d1974f401e1f
COVID-2390.png,COVID,COVID/images/COVID-2390.png,COVID/masks/COVID-2390.png,True,G01517,train,524d80bd76a3bf50a2ebcaf2491bde18,d42947c75e486f38
COVID-2391.png,COVID,COVID/images/COVID-2391.png,COVID/masks/COVID-2391.png,True,G01518,train,ed58688922537ebb1bb447c106e1d278,990f065945dae3c7
COVID-2392.png,COVID,COVID/images/COVID-2392.png,COVID/masks/COVID-2392.png,True,G01519,train,a1f60403f41d5c29dcd5552fe4d058bb,c92a97ca1f58591d
COVID-2393.png,COVID,COVID/images/COVID-2393.png,COVID/masks/COVID-2393.png,True,G01520,val,70232461350c123e2316c97662700e58,817906569a17cf5e
COVID-2394.png,COVID,COVID/images/COVID-2394.png,COVID/masks/COVID-2394.png,True,G01521,train,f4aa52e3cb39e0636873fca344803290,aa3c2b4b8f4abc68
COVID-2395.png,COVID,COVID/images/COVID-2395.png,COVID/masks/COVID-2395.png,True,G01522,val,b78a3a3d3d0858cf609adabbb890ca42,942e66d71155ef44
COVID-2396.png,COVID,COVID/images/COVID-2396.png,COVID/masks/COVID-2396.png,True,G01523,test,5fc7877d2d1fe9c0bd827a8ccef12eb4,d4afc3d10a4eaf41
COVID-2397.png,COVID,COVID/images/COVID-2397.png,COVID/masks/COVID-2397.png,True,G01524,test,db8d7a13883e1591c19f794c95bf6048,c07d87531b4f4738
COVID-2398.png,COVID,COVID/images/COVID-2398.png,COVID/masks/COVID-2398.png,True,G01525,test,2f34a28a604b311bdd4dbd6a821a8768,d95c145a7b7b9884
COVID-2399.png,COVID,COVID/images/COVID-2399.png,COVID/masks/COVID-2399.png,True,G01526,train,c9b412c0ddc653c376c9506f47aa40f2,d42c07571b1f071f
COVID-24.png,COVID,COVID/images/COVID-24.png,COVID/masks/COVID-24.png,True,G01527,train,ac7298b9704cf87442eb838b2e4c3c18,d42dc31d5c4c3e6a
COVID-240.png,COVID,COVID/images/COVID-240.png,COVID/masks/COVID-240.png,True,G01528,train,01daa8679543339f49f6d3320a01be8a,91680e1f798d7bc2
COVID-2400.png,COVID,COVID/images/COVID-2400.png,COVID/masks/COVID-2400.png,True,G01529,train,2aac3b64cc9c0886993500381d052575,817d061e5b696979
COVID-2401.png,COVID,COVID/images/COVID-2401.png,COVID/masks/COVID-2401.png,True,G01530,train,389e8095d2f9bd20f26e16e29a32eee7,817a5c187b4b69c7
COVID-2402.png,COVID,COVID/images/COVID-2402.png,COVID/masks/COVID-2402.png,True,G01531,train,9b56990777fea5983ca6a82cbef174e6,837a1c15325a7a7b
COVID-2403.png,COVID,COVID/images/COVID-2403.png,COVID/masks/COVID-2403.png,True,G01532,train,26e6a4ef09cae8edee68df56249cec20,812f7e1e7b312525
COVID-2404.png,COVID,COVID/images/COVID-2404.png,COVID/masks/COVID-2404.png,True,G01533,val,4dcba1572dc0bf6a3014dcc01ac9bc69,c12b3e16da5b3878
COVID-2405.png,COVID,COVID/images/COVID-2405.png,COVID/masks/COVID-2405.png,True,G01534,train,db84b5cd716a081dd799902b71e41380,dd2a131943797959
COVID-2406.png,COVID,COVID/images/COVID-2406.png,COVID/masks/COVID-2406.png,True,G01535,train,6a5c5e6a52043965d279428b58e441df,dd28c31f5d4c1a6c
COVID-2407.png,COVID,COVID/images/COVID-2407.png,COVID/masks/COVID-2407.png,True,G01536,test,fa3828c20bfc36d7751912e6033482c9,817f96184f1b591e
COVID-2408.png,COVID,COVID/images/COVID-2408.png,COVID/masks/COVID-2408.png,True,G01537,test,ef3efd2fd761e61b2d4b05596646a495,807947c656967e87
COVID-2409.png,COVID,COVID/images/COVID-2409.png,COVID/masks/COVID-2409.png,True,G01538,train,f8b3c2218ca7710bf7d7a832edb2813e,9959871e1c4a1f5e
COVID-241.png,COVID,COVID/images/COVID-241.png,COVID/masks/COVID-241.png,True,G00414,train,283c2b06960c7b8d07a67170d711b6be,d86d561e486d3c3c
COVID-2410.png,COVID,COVID/images/COVID-2410.png,COVID/masks/COVID-2410.png,True,G01539,train,39bf099c82f57dd6d570c741163d0a94,d02fc3161d4d6d3c
COVID-2411.png,COVID,COVID/images/COVID-2411.png,COVID/masks/COVID-2411.png,True,G01540,train,9e978642d96ddc0aa9cf048b92d00005,c42e874f1e4d4e5c
COVID-2412.png,COVID,COVID/images/COVID-2412.png,COVID/masks/COVID-2412.png,True,G01541,train,694529b9157806783e3ed75cb6e61222,817bc65e1ad45e31
COVID-2413.png,COVID,COVID/images/COVID-2413.png,COVID/masks/COVID-2413.png,True,G01542,train,bc8833297dd6b83530b3b8dfaa4c7a62,9131c6c619d6cf1b
COVID-2414.png,COVID,COVID/images/COVID-2414.png,COVID/masks/COVID-2414.png,True,G01543,train,93030915c37ece766a21502e32013980,d52f46471a1d4d4d
COVID-2415.png,COVID,COVID/images/COVID-2415.png,COVID/masks/COVID-2415.png,True,G01544,train,027ab20c974b344c597470b6225b776f,913b461e1a691b7d
COVID-2416.png,COVID,COVID/images/COVID-2416.png,COVID/masks/COVID-2416.png,True,G01545,val,0b287468e4b1df0d234306f25b1c4b80,d42b7b183cd1c7c4
COVID-2417.png,COVID,COVID/images/COVID-2417.png,COVID/masks/COVID-2417.png,True,G01546,val,ff584375bff797f0b0c3502a7b7fad75,817f6e0639277712
COVID-2418.png,COVID,COVID/images/COVID-2418.png,COVID/masks/COVID-2418.png,True,G01545,val,0b287468e4b1df0d234306f25b1c4b80,d42b7b183cd1c7c4
COVID-2419.png,COVID,COVID/images/COVID-2419.png,COVID/masks/COVID-2419.png,True,G01547,test,0cc538ef316d40f83bdbc873b9b9d491,9178841b3b1f785e
COVID-242.png,COVID,COVID/images/COVID-242.png,COVID/masks/COVID-242.png,True,G01548,train,6957f0ecb6c13646ead3a47109550e09,943e47193be56cd0
COVID-2420.png,COVID,COVID/images/COVID-2420.png,COVID/masks/COVID-2420.png,True,G01547,test,4e013270650f939b3b50c37d1d217337,9178841b3b1f785e
COVID-2421.png,COVID,COVID/images/COVID-2421.png,COVID/masks/COVID-2421.png,True,G01547,test,4e013270650f939b3b50c37d1d217337,9178841b3b1f785e
COVID-2422.png,COVID,COVID/images/COVID-2422.png,COVID/masks/COVID-2422.png,True,G01549,train,6927bed5f4e8c43b3df6e4d36e31e01f,812907571ec75f17
COVID-2423.png,COVID,COVID/images/COVID-2423.png,COVID/masks/COVID-2423.png,True,G01549,train,6927bed5f4e8c43b3df6e4d36e31e01f,812907571ec75f17
COVID-2424.png,COVID,COVID/images/COVID-2424.png,COVID/masks/COVID-2424.png,True,G01550,train,57f9368fc5a40fa08e0e02e4b068d3aa,817b06161a5f5f9a
COVID-2425.png,COVID,COVID/images/COVID-2425.png,COVID/masks/COVID-2425.png,True,G01551,train,cb94459cdf0646886267d58e50c4ead3,81a83e877b8d6b58
COVID-2426.png,COVID,COVID/images/COVID-2426.png,COVID/masks/COVID-2426.png,True,G01552,train,07b106a3e1e4b1c2cd7a7e7e27bf018d,d03b5a0839393f3d
COVID-2427.png,COVID,COVID/images/COVID-2427.png,COVID/masks/COVID-2427.png,True,G01553,train,ff4ec257acc2b9ff6e513b1f6d732ff2,d53c06c75f1b044f
COVID-2428.png,COVID,COVID/images/COVID-2428.png,COVID/masks/COVID-2428.png,True,G01554,train,acaa22d62ff1925b3a3a2975e8822c99,993f065a0d6939ec
COVID-2429.png,COVID,COVID/images/COVID-2429.png,COVID/masks/COVID-2429.png,True,G01555,train,a1d7f0d0da02d365d1cebbd22e0af0b0,817907861a1fde5b
COVID-243.png,COVID,COVID/images/COVID-243.png,COVID/masks/COVID-243.png,True,G01556,test,2ff0452b98e5bc20565c464d3b4b2615,d52a535b4f4e480f
COVID-2430.png,COVID,COVID/images/COVID-2430.png,COVID/masks/COVID-2430.png,True,G01557,train,6be539354ba81ac2663bd73a635c7ec0,817f860a155f5b1d
COVID-2431.png,COVID,COVID/images/COVID-2431.png,COVID/masks/COVID-2431.png,True,G01558,train,6a0c74f1833da20a3ff39c6a5b48df82,b96dc61b1d4f481c
COVID-2432.png,COVID,COVID/images/COVID-2432.png,COVID/masks/COVID-2432.png,True,G01558,train,cb57413d86da8811f3c495d81743d19f,b96dc61b1d4f481c
COVID-2433.png,COVID,COVID/images/COVID-2433.png,COVID/masks/COVID-2433.png,True,G01559,train,6718b126160b0ebb949e1ad933bd96b9,bc5c4709082bbbbe
COVID-2434.png,COVID,COVID/images/COVID-2434.png,COVID/masks/COVID-2434.png,True,G01560,train,64e6f324f5e352acd7f4ae435fe0cd9c,c93d56495c493d0f
COVID-2435.png,COVID,COVID/images/COVID-2435.png,COVID/masks/COVID-2435.png,True,G01561,val,8e22c11fcfb0b0b7d58ceb6efa2ded95,9f60c456165e5a57
COVID-2436.png,COVID,COVID/images/COVID-2436.png,COVID/masks/COVID-2436.png,True,G01562,val,3732eb32b5ab423620d5065ae1d982dc,8875c24bd34b5ed2
COVID-2437.png,COVID,COVID/images/COVID-2437.png,COVID/masks/COVID-2437.png,True,G01563,train,aff96a8df5f8107aa781a1555b09b08e,9168861297c67dd7
COVID-2438.png,COVID,COVID/images/COVID-2438.png,COVID/masks/COVID-2438.png,True,G01564,val,dc73281e0d8c0bfbc91dd01e2d4bbb1e,91a8061a1f5f595f
COVID-2439.png,COVID,COVID/images/COVID-2439.png,COVID/masks/COVID-2439.png,True,G01565,test,f960199bffc6982c58aabbddad65fb8a,917f84185b5b585b
COVID-244.png,COVID,COVID/images/COVID-244.png,COVID/masks/COVID-244.png,True,G01566,test,c3d0896dbb651df009018bebcc01957f,cd7cf243c3494559
COVID-2440.png,COVID,COVID/images/COVID-2440.png,COVID/masks/COVID-2440.png,True,G01567,train,115c114255242a2707a6b1a19bc8a4ab,dd2ac61f26c03bc5
COVID-2441.png,COVID,COVID/images/COVID-2441.png,COVID/masks/COVID-2441.png,True,G01568,train,f8f3d3477837961bf5a79abbfb2aa5bd,817d961a5d5a5969
COVID-2442.png,COVID,COVID/images/COVID-2442.png,COVID/masks/COVID-2442.png,True,G01569,train,f974a1b0586b47999d266576c9383634,d02d7ad2c34b5a59
COVID-2443.png,COVID,COVID/images/COVID-2443.png,COVID/masks/COVID-2443.png,True,G01570,train,00c6eef33608a9b3d8ff810b0d40d979,81b97e967e943a90
COVID-2444.png,COVID,COVID/images/COVID-2444.png,COVID/masks/COVID-2444.png,True,G01571,train,5f41b76ca40c5c2d38e60a90687fdcd0,d43d1a42cb6f4c5a
COVID-2445.png,COVID,COVID/images/COVID-2445.png,COVID/masks/COVID-2445.png,True,G01572,train,3a033c10e012b6f3b9be1a9e0f772169,d42ff5d199c28139
COVID-2446.png,COVID,COVID/images/COVID-2446.png,COVID/masks/COVID-2446.png,True,G01573,train,500e0f094e9c9b90087994692619eb44,897f87161a4d5f18
COVID-2447.png,COVID,COVID/images/COVID-2447.png,COVID/masks/COVID-2447.png,True,G01574,train,cbd71ce705ceac408e417ef21867d8dd,887f87161e495d1d
COVID-2448.png,COVID,COVID/images/COVID-2448.png,COVID/masks/COVID-2448.png,True,G01575,train,0fe59698133b098b42917ab8310db4b9,d42e425f0b4c6f3c
COVID-2449.png,COVID,COVID/images/COVID-2449.png,COVID/masks/COVID-2449.png,True,G01576,train,74182e7d0efd12cf90093434e12cf7ef,81e857967a857a99
COVID-245.png,COVID,COVID/images/COVID-245.png,COVID/masks/COVID-245.png,True,G01577,train,ad32d686ab288d35e580dd8b33820df2,d42c0f0a0f7e2d7c
COVID-2450.png,COVID,COVID/images/COVID-2450.png,COVID/masks/COVID-2450.png,True,G01578,train,375a4dc43c3ee41d962b21002626091a,802dd7065e4f5b5a
COVID-2451.png,COVID,COVID/images/COVID-2451.png,COVID/masks/COVID-2451.png,True,G01579,train,4476024937a90cd8c4c35106b1ae52db,8178561e1a6d7b39
COVID-2452.png,COVID,COVID/images/COVID-2452.png,COVID/masks/COVID-2452.png,True,G01580,val,1779f3137ce0aa2d3ce0513963c5a842,d43fc3494c4e2e2e
COVID-2453.png,COVID,COVID/images/COVID-2453.png,COVID/masks/COVID-2453.png,True,G01581,train,e4198f010ce604a0155f209e7d85933c,817b8f141a8a5f4f
COVID-2454.png,COVID,COVID/images/COVID-2454.png,COVID/masks/COVID-2454.png,True,G01582,val,1ab7e7d5bb5ee181449c817d757df55f,c475d34b820bb47e
COVID-2455.png,COVID,COVID/images/COVID-2455.png,COVID/masks/COVID-2455.png,True,G01583,val,524b2f9b2dc8e267c5fd6b05eead2c68,cd2e54f2520b4f55
COVID-2456.png,COVID,COVID/images/COVID-2456.png,COVID/masks/COVID-2456.png,True,G01584,test,149cdbe0e6a1f26b36397bf0c8f5a750,dd2f44821b0d5d5d
COVID-2457.png,COVID,COVID/images/COVID-2457.png,COVID/masks/COVID-2457.png,True,G01585,val,acd2991b965ca73f621ec032e2d86d69,d42c50f39b43d34b
COVID-2458.png,COVID,COVID/images/COVID-2458.png,COVID/masks/COVID-2458.png,True,G01586,test,21f8a38e311cd46fdc3df0e118c0892d,c42ec7c30d0e5f4d
COVID-2459.png,COVID,COVID/images/COVID-2459.png,COVID/masks/COVID-2459.png,True,G01586,test,21f8a38e311cd46fdc3df0e118c0892d,c42ec7c30d0e5f4d
COVID-246.png,COVID,COVID/images/COVID-246.png,COVID/masks/COVID-246.png,True,G00411,train,d2a409269d5c3e26883f722e5e6c8cb1,cc4b530f4d2d8fa8
COVID-2460.png,COVID,COVID/images/COVID-2460.png,COVID/masks/COVID-2460.png,True,G01587,train,0e16fc93a6ca132be19abd1b34e37d1c,9d1c421a1aed78fc
COVID-2461.png,COVID,COVID/images/COVID-2461.png,COVID/masks/COVID-2461.png,True,G01588,test,bfd624cb2e5da97b24c322e17ed36705,917846131b4f1b7b
COVID-2462.png,COVID,COVID/images/COVID-2462.png,COVID/masks/COVID-2462.png,True,G01589,train,f9c54a3d4b9e70a559f2b9dc218c6297,995d161e491b495f
COVID-2463.png,COVID,COVID/images/COVID-2463.png,COVID/masks/COVID-2463.png,True,G01590,test,1ced25be8f4775f89e6898ad599ed877,d428d34b5e5a1f4c
COVID-2464.png,COVID,COVID/images/COVID-2464.png,COVID/masks/COVID-2464.png,True,G01591,train,e69663e7e2fad5532f029087e2a657b0,80bc8f015e5e7e5a
COVID-2465.png,COVID,COVID/images/COVID-2465.png,COVID/masks/COVID-2465.png,True,G01592,train,7282f714d9d928e266416c31c1d8bacb,9b611c5f7a806ba3
COVID-2466.png,COVID,COVID/images/COVID-2466.png,COVID/masks/COVID-2466.png,True,G01593,val,1038dcf3b145217afec63e0f05dd3607,c834431c4b6fb96d
COVID-2467.png,COVID,COVID/images/COVID-2467.png,COVID/masks/COVID-2467.png,True,G01594,train,4e6cabb64a31604085cdd49cdef83b3a,dd2ad7031e45635c
COVID-2468.png,COVID,COVID/images/COVID-2468.png,COVID/masks/COVID-2468.png,True,G01595,train,36877803f8f4f213a8c3198bc940bbd4,cd0a514f4f4b0f2b
COVID-2469.png,COVID,COVID/images/COVID-2469.png,COVID/masks/COVID-2469.png,True,G01596,train,5cb97d4d9ddc53e1eec5ba9787d4506b,dd2ed0494d454b5d
COVID-247.png,COVID,COVID/images/COVID-247.png,COVID/masks/COVID-247.png,True,G00255,train,f7e6e500001e97ebc1931176a1a9f167,dd25d20fd20bd668
COVID-2470.png,COVID,COVID/images/COVID-2470.png,COVID/masks/COVID-2470.png,True,G01597,train,70928a44419ec36b849954a187cf40fd,817d87061e1e4f1f
COVID-2471.png,COVID,COVID/images/COVID-2471.png,COVID/masks/COVID-2471.png,True,G01598,train,2bc182349b1c95e9cf360ca26a6bcdc9,987f461c596ba9a4
COVID-2472.png,COVID,COVID/images/COVID-2472.png,COVID/masks/COVID-2472.png,True,G01599,train,a389ca84e1a909cf3c18f674da28d159,f97d86d71a134701
COVID-2473.png,COVID,COVID/images/COVID-2473.png,COVID/masks/COVID-2473.png,True,G01600,val,540274fdfe2eb81cb1d530f1a7db5a20,8d2e83c14f4d4e5d
COVID-2474.png,COVID,COVID/images/COVID-2474.png,COVID/masks/COVID-2474.png,True,G01601,train,ab884a65193f7a79ab45f04651ccd79d,9c6fc7131d41535a
COVID-2475.png,COVID,COVID/images/COVID-2475.png,COVID/masks/COVID-2475.png,True,G01602,train,dde4dea1c2ff228d7366575746b767b3,8c6ec35bc9c0dac3
COVID-2476.png,COVID,COVID/images/COVID-2476.png,COVID/masks/COVID-2476.png,True,G01603,train,9a84117d5ccafad0d8a3b6502f3c5288,9d6ec34b4d543c49
COVID-2477.png,COVID,COVID/images/COVID-2477.png,COVID/masks/COVID-2477.png,True,G01604,train,f0e3cbfb11726d4b829dd4f7f8cd45f1,b919841e5b597c5c
COVID-2478.png,COVID,COVID/images/COVID-2478.png,COVID/masks/COVID-2478.png,True,G01605,test,ef8e2e533fb2e1514837651805d49f86,887e6ac79e58c9c1
COVID-2479.png,COVID,COVID/images/COVID-2479.png,COVID/masks/COVID-2479.png,True,G01606,test,621498e8f895f87bd7c0a505d2075db6,d97f86161a0a5f19
COVID-248.png,COVID,COVID/images/COVID-248.png,COVID/masks/COVID-248.png,True,G01607,train,2ada1bd7b96e04ca1c7e69619cf820eb,d42c0f0b0f6a3d7c
COVID-2480.png,COVID,COVID/images/COVID-2480.png,COVID/masks/COVID-2480.png,True,G01608,train,ed0f5083b1220f2fce02e3e946002556,817f0786161d5d1f
COVID-2481.png,COVID,COVID/images/COVID-2481.png,COVID/masks/COVID-2481.png,True,G01609,train,b35245d3da641cb5305e82ffe0cf3b35,817b07061a5b5f1f
COVID-2482.png,COVID,COVID/images/COVID-2482.png,COVID/masks/COVID-2482.png,True,G01610,train,a1fabf50cb3c917d465cc8d0f04f47f3,897f161a5b7839a1
COVID-2483.png,COVID,COVID/images/COVID-2483.png,COVID/masks/COVID-2483.png,True,G01611,train,199df7766179625659cab3310fe42061,897d065a5b4c78e9
COVID-2484.png,COVID,COVID/images/COVID-2484.png,COVID/masks/COVID-2484.png,True,G01612,train,c045030a2fee322cc23570e69411cc0e,896e065a4d7cf8e4
COVID-2485.png,COVID,COVID/images/COVID-2485.png,COVID/masks/COVID-2485.png,True,G01613,train,7f23a71cfde035dde5723c6f1dd1848e,d43c838b1bce4f38
COVID-2486.png,COVID,COVID/images/COVID-2486.png,COVID/masks/COVID-2486.png,True,G01614,train,79b99eeb721f8d96750693949f382114,dd3cd24b544b1d43
COVID-2487.png,COVID,COVID/images/COVID-2487.png,COVID/masks/COVID-2487.png,True,G01615,train,3946510f3a9a7d1e05213b25d60131be,c43d4b5b13c32d1b
COVID-2488.png,COVID,COVID/images/COVID-2488.png,COVID/masks/COVID-2488.png,True,G01616,train,f6f831cf838b6a8662c020fa2c2f72e7,dd2f82194d4f194e
COVID-2489.png,COVID,COVID/images/COVID-2489.png,COVID/masks/COVID-2489.png,True,G01617,train,e4fbc31de8f2d9317c90a0a606935836,d87f871a4b1a1e45
COVID-249.png,COVID,COVID/images/COVID-249.png,COVID/masks/COVID-249.png,True,G01618,train,b199af3748b2e19ec8a76ffe4eb9460b,dd28c20b2f692d6d
COVID-2490.png,COVID,COVID/images/COVID-2490.png,COVID/masks/COVID-2490.png,True,G01619,val,36d3abafc4039c2ff16d17c40ea74869,817b85161a1d5f5b
COVID-2491.png,COVID,COVID/images/COVID-2491.png,COVID/masks/COVID-2491.png,True,G01620,val,ca1132588172ea2ec7bc9aa462d494ab,d57dc6430ad6825b
COVID-2492.png,COVID,COVID/images/COVID-2492.png,COVID/masks/COVID-2492.png,True,G01621,train,3ba722d1d93f3f80763c46b8c7af5599,dc2e43430c2cf5fc
COVID-2493.png,COVID,COVID/images/COVID-2493.png,COVID/masks/COVID-2493.png,True,G01621,train,3ba722d1d93f3f80763c46b8c7af5599,dc2e43430c2cf5fc
COVID-2494.png,COVID,COVID/images/COVID-2494.png,COVID/masks/COVID-2494.png,True,G01622,train,3429bd5859b2547aeb44b46acc0bdd06,813d468e1a5b4b7d
COVID-2495.png,COVID,COVID/images/COVID-2495.png,COVID/masks/COVID-2495.png,True,G01623,test,aed69d23aa59b57b4b47c1d28c9df2a4,89790616195b7b5b
COVID-2496.png,COVID,COVID/images/COVID-2496.png,COVID/masks/COVID-2496.png,True,G01624,test,742c5419a54cc5ac85fe857a06dcd5d2,d47c178b4f0d0e55
COVID-2497.png,COVID,COVID/images/COVID-2497.png,COVID/masks/COVID-2497.png,True,G01625,train,7ae4d49dc2ead766f3ad9be6d99e178a,91a96ac43e963ad3
COVID-2498.png,COVID,COVID/images/COVID-2498.png,COVID/masks/COVID-2498.png,True,G01626,train,1c38aef563f9cf6fb4151abc7d5d937b,c0e937d24bd6164b
COVID-2499.png,COVID,COVID/images/COVID-2499.png,COVID/masks/COVID-2499.png,True,G01627,val,50d84478a466167ed05c9ec6cbf5dbed,95384f071e58597d
COVID-25.png,COVID,COVID/images/COVID-25.png,COVID/masks/COVID-25.png,True,G01628,test,16f065a3d3b6b66a07fbeb92a187b3c3,9978861b195b4d57
COVID-250.png,COVID,COVID/images/COVID-250.png,COVID/masks/COVID-250.png,True,G00355,train,953fcf78eef0297e36c9a35fbf3a375f,8835d20fd319577c
COVID-2500.png,COVID,COVID/images/COVID-2500.png,COVID/masks/COVID-2500.png,True,G01629,train,014aa1ca8db505e709bb16fb5d7fd79e,817a5418597fe8f8
COVID-2501.png,COVID,COVID/images/COVID-2501.png,COVID/masks/COVID-2501.png,True,G01630,train,f03e3011bf62bb715797c9266838b169,dd28521a082dafbf
COVID-2502.png,COVID,COVID/images/COVID-2502.png,COVID/masks/COVID-2502.png,True,G01631,train,8a936dbcb483e52c5cff9c82eafa11dc,dc2a430b0b4f2f3d
COVID-2503.png,COVID,COVID/images/COVID-2503.png,COVID/masks/COVID-2503.png,True,G01632,train,ded0bb41f73e9d5f97fca4c46ca66169,dd2a424b0a2ded3d
COVID-2504.png,COVID,COVID/images/COVID-2504.png,COVID/masks/COVID-2504.png,True,G01633,train,71da5362894f197a90ae70bec412e0c0,dc6e430b0bad0cbd
COVID-2505.png,COVID,COVID/images/COVID-2505.png,COVID/masks/COVID-2505.png,True,G01634,train,e13abb80fb2abbd6ec9df72fd18042b5,917bc65e1b481f38
COVID-2506.png,COVID,COVID/images/COVID-2506.png,COVID/masks/COVID-2506.png,True,G01635,val,09ea037f77cc8b2d678574f314f7aa77,d959d61e1e0a1a3e
COVID-2507.png,COVID,COVID/images/COVID-2507.png,COVID/masks/COVID-2507.png,True,G01636,val,77930f89abceb0631fcc3f8f262831aa,9879c71e145b1a5e
COVID-2508.png,COVID,COVID/images/COVID-2508.png,COVID/masks/COVID-2508.png,True,G01637,train,c863436d499f4bcc4a768dae8c30ebb5,997947c61647571a
COVID-2509.png,COVID,COVID/images/COVID-2509.png,COVID/masks/COVID-2509.png,True,G01638,test,5cc4ce46307f390f0e33ee3c4842dcb9,99688619855e5b9f
COVID-251.png,COVID,COVID/images/COVID-251.png,COVID/masks/COVID-251.png,True,G01639,train,e837f6a296a8892fe80665586160bd3f,817807c63ec46ed7
COVID-2510.png,COVID,COVID/images/COVID-2510.png,COVID/masks/COVID-2510.png,True,G01640,train,6decbbe1df7643fd0cc659495727b899,817846181f4d7b3f
COVID-2511.png,COVID,COVID/images/COVID-2511.png,COVID/masks/COVID-2511.png,True,G01641,test,e2f7595fa4d62ff221304dbf5ae243fb,d12b461e1b4d3d39
COVID-2512.png,COVID,COVID/images/COVID-2512.png,COVID/masks/COVID-2512.png,True,G01642,train,8390111c2a72ff88d46190b44b5745fd,dc2cc65a4c4f186f
COVID-2513.png,COVID,COVID/images/COVID-2513.png,COVID/masks/COVID-2513.png,True,G01643,train,2ed438d7bae9e34b026a7fb2850acb52,903fc7561ac74f18
COVID-2514.png,COVID,COVID/images/COVID-2514.png,COVID/masks/COVID-2514.png,True,G01644,train,5c0211118b82e66db6996a9213f8f9d5,d43f42431b594f5c
COVID-2515.png,COVID,COVID/images/COVID-2515.png,COVID/masks/COVID-2515.png,True,G01645,val,8e430040169684886df5774473609fa5,dc7ad34f0c2a2c27
COVID-2516.png,COVID,COVID/images/COVID-2516.png,COVID/masks/COVID-2516.png,True,G01646,test,2a3bc3bbb1c588dababbf18e2feadf44,dc2ef1d1594dc845
COVID-2517.png,COVID,COVID/images/COVID-2517.png,COVID/masks/COVID-2517.png,True,G01647,train,d36998a6c8048316e6585cecb6774ae9,997bd65a58191b1a
COVID-2518.png,COVID,COVID/images/COVID-2518.png,COVID/masks/COVID-2518.png,True,G01648,train,7a0a1a5acd2fcb61f88d20f49c2edc5d,d52f4b4d4e4a6a4a
COVID-2519.png,COVID,COVID/images/COVID-2519.png,COVID/masks/COVID-2519.png,True,G01649,train,bda060585c214466a532381f86b466fb,817f174a5b39a469
COVID-252.png,COVID,COVID/images/COVID-252.png,COVID/masks/COVID-252.png,True,G01650,train,0e362bed4541ae1fb1e591c2b829d375,dcacd25bd1525d42
COVID-2520.png,COVID,COVID/images/COVID-2520.png,COVID/masks/COVID-2520.png,True,G01651,train,0bd97044a1ec6184b9a9c800215839a3,d87bd25a58527916
COVID-2521.png,COVID,COVID/images/COVID-2521.png,COVID/masks/COVID-2521.png,True,G01652,train,1dd0b2e2f969b91b63afaf68dd74a3f8,997e07194d4e194f
COVID-2522.png,COVID,COVID/images/COVID-2522.png,COVID/masks/COVID-2522.png,True,G01653,train,97fa568d8863149852a8293751fda399,846c071f597f287c
COVID-2523.png,COVID,COVID/images/COVID-2523.png,COVID/masks/COVID-2523.png,True,G01654,train,374578c0e5f146006c1c317e022e878d,d42ce393514b475d
COVID-2524.png,COVID,COVID/images/COVID-2524.png,COVID/masks/COVID-2524.png,True,G01655,train,6cee250699d4f04e8971ead108da8539,817f82d61a115f5d
COVID-2525.png,COVID,COVID/images/COVID-2525.png,COVID/masks/COVID-2525.png,True,G01656,train,89e733c52df64e42cbee3fb54775a703,dd0a534a5f0a4b2f
COVID-2526.png,COVID,COVID/images/COVID-2526.png,COVID/masks/COVID-2526.png,True,G01657,train,3719ae76832d19ccc189622cd5566feb,817f465e18495d79
COVID-2527.png,COVID,COVID/images/COVID-2527.png,COVID/masks/COVID-2527.png,True,G01658,train,4b4b12dd40c76dc018c9460d5a9813e1,952e071f194d5c7c
COVID-2528.png,COVID,COVID/images/COVID-2528.png,COVID/masks/COVID-2528.png,True,G01659,train,f83a2d98fff8cbe100308a355df8aec1,817987941adece1e
COVID-2529.png,COVID,COVID/images/COVID-2529.png,COVID/masks/COVID-2529.png,True,G01660,val,368ebe34e0a29096422ee658908303fb,996f06584d38edb8
COVID-253.png,COVID,COVID/images/COVID-253.png,COVID/masks/COVID-253.png,True,G01661,val,5d695d655aef0c299ade9204e183bd1e,9d2cc41bb2529f56
COVID-2530.png,COVID,COVID/images/COVID-2530.png,COVID/masks/COVID-2530.png,True,G01662,train,b5ecdd37d24bb54b4f11ef808542abc4,997f061a594d784d
COVID-2531.png,COVID,COVID/images/COVID-2531.png,COVID/masks/COVID-2531.png,True,G01663,train,7316a56564ac86b2066a9991a9290507,91684e061a5f7e79
COVID-2532.png,COVID,COVID/images/COVID-2532.png,COVID/masks/COVID-2532.png,True,G01664,val,0353924a8c5d63c3b487f82ebdf821e2,d42ee2d79d0ac383
COVID-2533.png,COVID,COVID/images/COVID-2533.png,COVID/masks/COVID-2533.png,True,G01665,test,bb35bb187b18f46ca07c4e6013c45ebd,d86d449613575e59
COVID-2534.png,COVID,COVID/images/COVID-2534.png,COVID/masks/COVID-2534.png,True,G01666,test,0519b631290bb3060f703aeb20111ab0,dc3d4b0e082f2e2f
COVID-2535.png,COVID,COVID/images/COVID-2535.png,COVID/masks/COVID-2535.png,True,G01667,test,b023b769b857f58e9893e607140ba309,d07dc7121a2f0e3e
COVID-2536.png,COVID,COVID/images/COVID-2536.png,COVID/masks/COVID-2536.png,True,G01668,train,a8a2151e3dd73c8e2da047405b5b8fd9,d4dc2ad1296fc529
COVID-2537.png,COVID,COVID/images/COVID-2537.png,COVID/masks/COVID-2537.png,True,G01669,train,2def46bb674b912ab9323d9aff1d2631,dd2d96529b494d1c
COVID-2538.png,COVID,COVID/images/COVID-2538.png,COVID/masks/COVID-2538.png,True,G01670,train,1aa0e17b50b43a74709cdfbbd9092878,d52d16865f0d0f1d
COVID-2539.png,COVID,COVID/images/COVID-2539.png,COVID/masks/COVID-2539.png,True,G01671,train,81870dcfdf51c0e7f60ae5ac612096f6,d52d025b28673f74
COVID-254.png,COVID,COVID/images/COVID-254.png,COVID/masks/COVID-254.png,True,G01672,val,760fecad41a4c80abc61682b92154b52,d47c0f030f5b6c2d
COVID-2540.png,COVID,COVID/images/COVID-2540.png,COVID/masks/COVID-2540.png,True,G01673,train,26b5fd692cc3638d29232ac17abc2978,807f465a48cd5f1d
COVID-2541.png,COVID,COVID/images/COVID-2541.png,COVID/masks/COVID-2541.png,True,G01674,val,7083df1340d299c1f2aa9f3185a58ba4,95ac06451f0f0f5f
COVID-2542.png,COVID,COVID/images/COVID-2542.png,COVID/masks/COVID-2542.png,True,G01675,test,c2e713ea25ba1f8ccee63c58023633d3,c13cc71f134f1a1b
COVID-2543.png,COVID,COVID/images/COVID-2543.png,COVID/masks/COVID-2543.png,True,G01676,train,afc0532a1f717dda7983ad2b2afcfc7c,d43dc351430fb84f
COVID-2544.png,COVID,COVID/images/COVID-2544.png,COVID/masks/COVID-2544.png,True,G01677,train,0fb72d5bb2b0fd5a8d8835c42fc757d4,959166967a7a0d6a
COVID-2545.png,COVID,COVID/images/COVID-2545.png,COVID/masks/COVID-2545.png,True,G01678,train,58a1ecde55fd9af4274cb6e9880c514f,817d06861f1b1cef
COVID-2546.png,COVID,COVID/images/COVID-2546.png,COVID/masks/COVID-2546.png,True,G01679,train,b975fb38e3f7eaf444dfc68c7c52f5c6,817b56521a5b4f1b
COVID-2547.png,COVID,COVID/images/COVID-2547.png,COVID/masks/COVID-2547.png,True,G01680,train,dbb9e40474aa759a70ed10fd32f7a960,d42c63cb2ec86bd8
COVID-2548.png,COVID,COVID/images/COVID-2548.png,COVID/masks/COVID-2548.png,True,G01681,train,cdbe712fc54c6be2d2df9737fca7e698,d42e434f0a6f2c7c
COVID-2549.png,COVID,COVID/images/COVID-2549.png,COVID/masks/COVID-2549.png,True,G01682,train,846dfd76c2c06819b0bc7b2b8c86cdcf,cd0d404b4d4d5f6e
COVID-255.png,COVID,COVID/images/COVID-255.png,COVID/masks/COVID-255.png,True,G01683,train,91a533df29b339164bc514a0ae2ee365,987f041a155e795b
COVID-2550.png,COVID,COVID/images/COVID-2550.png,COVID/masks/COVID-2550.png,True,G01684,train,f48af373e77f9153fbabc7c788526b61,807f861a0d3d3e6d
COVID-2551.png,COVID,COVID/images/COVID-2551.png,COVID/masks/COVID-2551.png,True,G01685,test,b332607889f9c4e10d52fe9025716891,8c74434b1b4f2d1f
COVID-2552.png,COVID,COVID/images/COVID-2552.png,COVID/masks/COVID-2552.png,True,G01686,train,1104860a0d5c11019d116b4d76413354,81a966c77b926ec4
COVID-2553.png,COVID,COVID/images/COVID-2553.png,COVID/masks/COVID-2553.png,True,G01687,val,283f4ed612002837eb7e233bbbe3cd35,917e470b1b4d6c78
COVID-2554.png,COVID,COVID/images/COVID-2554.png,COVID/masks/COVID-2554.png,True,G01688,test,095a91b8caba62cafb4ad20d191f7105,d03d4f431a1b4f0f
COVID-2555.png,COVID,COVID/images/COVID-2555.png,COVID/masks/COVID-2555.png,True,G01689,train,a97a42bf7e9e70a7eb542bff1a7d4696,c03c6f4f5b0b2c2e
COVID-2556.png,COVID,COVID/images/COVID-2556.png,COVID/masks/COVID-2556.png,True,G01690,train,3a124dd4d1be2ebea6108c3966423f0b,d0bd35d02bd03d5a
COVID-2557.png,COVID,COVID/images/COVID-2557.png,COVID/masks/COVID-2557.png,True,G01691,train,4f004626eefa5819a18632861bbab510,d0d03d27c70f2fd0
COVID-2558.png,COVID,COVID/images/COVID-2558.png,COVID/masks/COVID-2558.png,True,G01692,train,533814509d91da5a5e123e57514c0301,806a15c786577f1b
COVID-2559.png,COVID,COVID/images/COVID-2559.png,COVID/masks/COVID-2559.png,True,G01693,train,abae94d7222588f9877c38ab6cfe38a3,857e86841d7c1f5c
COVID-256.png,COVID,COVID/images/COVID-256.png,COVID/masks/COVID-256.png,True,G00098,train,0a07d8ac5e58bc1c644f84ba7717919c,f969c6165e461a6a
COVID-2560.png,COVID,COVID/images/COVID-2560.png,COVID/masks/COVID-2560.png,True,G01694,train,968ae0fa3ece72d6ff6bbd7c6e0b533a,9c2c07570b4f4f5a
COVID-2561.png,COVID,COVID/images/COVID-2561.png,COVID/masks/COVID-2561.png,True,G01695,train,751c6d0ddc40108aec6eab5bdb515493,992d87160a4f1e5f
COVID-2562.png,COVID,COVID/images/COVID-2562.png,COVID/masks/COVID-2562.png,True,G01696,train,8fbcc2627023fbe4363d6ecc4dc551f9,d42a034f4c5e6e6e
COVID-2563.png,COVID,COVID/images/COVID-2563.png,COVID/masks/COVID-2563.png,True,G01697,train,a9ea3f90f889076bbed51c42c302f5eb,d42e434b0a3f2f2d
COVID-2564.png,COVID,COVID/images/COVID-2564.png,COVID/masks/COVID-2564.png,True,G01698,train,eee56fd57d1a348e6fa20fce6ab74238,812a7e1979c7789c
COVID-2565.png,COVID,COVID/images/COVID-2565.png,COVID/masks/COVID-2565.png,True,G01699,val,33f977b01df4f473a8ebd038ab12ac68,d47d42cf2ac22bd8
COVID-2566.png,COVID,COVID/images/COVID-2566.png,COVID/masks/COVID-2566.png,True,G01700,train,dd578f21da13abb1aea542a41d9af1ff,d53c431f1c4f1b19
COVID-2567.png,COVID,COVID/images/COVID-2567.png,COVID/masks/COVID-2567.png,True,G01701,train,59cf3fbbbec0d4920dd160e8b405f918,d52ac3d71f43450d
COVID-2568.png,COVID,COVID/images/COVID-2568.png,COVID/masks/COVID-2568.png,True,G01702,train,3ed6315203eeeaf59c83b92b1b2abb38,9979871e5a4b3859
COVID-2569.png,COVID,COVID/images/COVID-2569.png,COVID/masks/COVID-2569.png,True,G01703,val,e23a4f69bdafc3acc536b7528ba7ca3a,9d7c40971ac3c31f
COVID-257.png,COVID,COVID/images/COVID-257.png,COVID/masks/COVID-257.png,True,G01704,val,cfea57004eb4a25f750d5460e8d274e5,d08303fe1bb5c2f4
COVID-2570.png,COVID,COVID/images/COVID-2570.png,COVID/masks/COVID-2570.png,True,G01705,test,4498c68a1400dd4803e9021753c2b8b4,917946c65e5e5e41
COVID-2571.png,COVID,COVID/images/COVID-2571.png,COVID/masks/COVID-2571.png,True,G01706,test,885418c0c04ea63ecc7a84093ae6eb88,987b45061e59497f
COVID-2572.png,COVID,COVID/images/COVID-2572.png,COVID/masks/COVID-2572.png,True,G01707,test,4ee21ee2a2afe5b31b7b72a646f4f7db,887b85165e195d5b
COVID-2573.png,COVID,COVID/images/COVID-2573.png,COVID/masks/COVID-2573.png,True,G01708,train,0f0a89b9b2e53fd1b9eb6091f048d2f1,9839851e1a595f1f
COVID-2574.png,COVID,COVID/images/COVID-2574.png,COVID/masks/COVID-2574.png,True,G01709,train,a58eccd44ab7b45d89479e5240767380,996bc61ed10f1a55
COVID-2575.png,COVID,COVID/images/COVID-2575.png,COVID/masks/COVID-2575.png,True,G01710,train,d12246e4a493c26458e8c21c3369cf2e,dd2cc61f93471f10
COVID-2576.png,COVID,COVID/images/COVID-2576.png,COVID/masks/COVID-2576.png,True,G01711,val,fa992a8c28c04f1dd956b7fc781a145c,8178c65a1f455a7b
COVID-2577.png,COVID,COVID/images/COVID-2577.png,COVID/masks/COVID-2577.png,True,G01712,train,2a9f940c67a66a201ad203eaa417324e,8178561e1b4d7b39
COVID-2578.png,COVID,COVID/images/COVID-2578.png,COVID/masks/COVID-2578.png,True,G01713,train,ceabf7d50e4a99fd758a109719b1a087,9169ce061b0f5e3b
COVID-2579.png,COVID,COVID/images/COVID-2579.png,COVID/masks/COVID-2579.png,True,G01714,train,1408b1980239346638d8f55114c501de,913c0e1a5b4f7879
COVID-258.png,COVID,COVID/images/COVID-258.png,COVID/masks/COVID-258.png,True,G01715,train,fb7b5c9e74844742c5598bff05a4ca21,d4f46e83291b8b2d
COVID-2580.png,COVID,COVID/images/COVID-2580.png,COVID/masks/COVID-2580.png,True,G01716,train,86b49898d42b28d66006198607848676,837ac4101b5f7a3b
COVID-2581.png,COVID,COVID/images/COVID-2581.png,COVID/masks/COVID-2581.png,True,G01717,train,6896b9a3517a54325663181ad98d9fbf,892d07165a4acf5f
COVID-2582.png,COVID,COVID/images/COVID-2582.png,COVID/masks/COVID-2582.png,True,G01718,train,102036fefa9b98b23bbfb0ff2aa6acd0,996f861acf11ba1c
COVID-2583.png,COVID,COVID/images/COVID-2583.png,COVID/masks/COVID-2583.png,True,G01719,train,97dd650e34b516ed85f6099820d26817,98e7d65a59585358
COVID-2584.png,COVID,COVID/images/COVID-2584.png,COVID/masks/COVID-2584.png,True,G01720,train,d3dfcc76d5236b69e2fa6ceb3f8ad4e3,b97f965a0d194b11
COVID-2585.png,COVID,COVID/images/COVID-2585.png,COVID/masks/COVID-2585.png,True,G01721,train,b5ddae86da6565a5efc7bbfc9981f669,d53cb0438393add5
COVID-2586.png,COVID,COVID/images/COVID-2586.png,COVID/masks/COVID-2586.png,True,G01722,train,a53ab128a5f69324cff1244053604096,d468c31b497d585d
COVID-2587.png,COVID,COVID/images/COVID-2587.png,COVID/masks/COVID-2587.png,True,G01723,val,26361992be100572313176de34c49e96,d93f861a1d125b5c
COVID-2588.png,COVID,COVID/images/COVID-2588.png,COVID/masks/COVID-2588.png,True,G01724,val,dec9f3262862bac09322e1e48f3e46af,d42dc3cb0f6e4a2c
COVID-2589.png,COVID,COVID/images/COVID-2589.png,COVID/masks/COVID-2589.png,True,G01725,test,03254a433d244ed073d77c9a3187dceb,d43dd2c30fce4ac8
COVID-259.png,COVID,COVID/images/COVID-259.png,COVID/masks/COVID-259.png,True,G01726,test,2b62940f0b602d63aa0f49153fa50199,dd28d3561e4e281f
COVID-2590.png,COVID,COVID/images/COVID-2590.png,COVID/masks/COVID-2590.png,True,G01727,test,e5420a67612f970dbd87f20ead8dd2df,817b47421e5c5f53
COVID-2591.png,COVID,COVID/images/COVID-2591.png,COVID/masks/COVID-2591.png,True,G01728,train,2ee04717172a55e77e1a37a2c1b72f64,98ae8619656b7c72
COVID-2592.png,COVID,COVID/images/COVID-2592.png,COVID/masks/COVID-2592.png,True,G01729,train,019b1cede7c87e72426863b7062e9cfa,817b841e195b7b3a
COVID-2593.png,COVID,COVID/images/COVID-2593.png,COVID/masks/COVID-2593.png,True,G01730,train,b9ec9bef22442c61f513ff3aec6614e3,c46e470b4f6c7968
COVID-2594.png,COVID,COVID/images/COVID-2594.png,COVID/masks/COVID-2594.png,True,G01731,train,8d6b4d0e5eebf116840ec16b0cffc76e,d42853534e2c7d3d
COVID-2595.png,COVID,COVID/images/COVID-2595.png,COVID/masks/COVID-2595.png,True,G01732,train,16e0c4bdd943e95c271b6ac0a6f7407c,cc2c434b6cedbcc4
COVID-2596.png,COVID,COVID/images/COVID-2596.png,COVID/masks/COVID-2596.png,True,G01733,train,dbf7e9fba901986f50cd5256646da990,d43cc3535f4a0b2d
COVID-2597.png,COVID,COVID/images/COVID-2597.png,COVID/masks/COVID-2597.png,True,G01734,train,35bdcec9b73162af47bee6154e437576,d42c425b1d4e3b3d
COVID-2598.png,COVID,COVID/images/COVID-2598.png,COVID/masks/COVID-2598.png,True,G01735,train,9f2d9f7f13fc304f018e2829c81c8be9,987985071e1f1f1e
COVID-2599.png,COVID,COVID/images/COVID-2599.png,COVID/masks/COVID-2599.png,True,G01736,train,0bd7e88a75587c5757adacf3991fbb82,983f841e1c5e1a5f
COVID-26.png,COVID,COVID/images/COVID-26.png,COVID/masks/COVID-26.png,True,G01737,train,89a1a45e13a098ce3e23aa573d9c73bd,d86196479487e597
COVID-260.png,COVID,COVID/images/COVID-260.png,COVID/masks/COVID-260.png,True,G01738,val,51b1582e2ce374d9be8eefcdf0548bc1,d43fd345584a785a
COVID-2600.png,COVID,COVID/images/COVID-2600.png,COVID/masks/COVID-2600.png,True,G01739,train,0b204f45b798773417e7e164268a646d,dc3cd35b424f08ce
COVID-2601.png,COVID,COVID/images/COVID-2601.png,COVID/masks/COVID-2601.png,True,G01740,train,b38fcb546773330fda235f91ad5bc76f,8979871e154f1f19
COVID-2602.png,COVID,COVID/images/COVID-2602.png,COVID/masks/COVID-2602.png,True,G01741,val,78067e0b74e599b8f704e05c71b71e26,d466c74b0e4f6a68
COVID-2603.png,COVID,COVID/images/COVID-2603.png,COVID/masks/COVID-2603.png,True,G01742,test,7948285f5eb04bb5bf69e12d0b2f5d5c,d07d630f083d391f
COVID-2604.png,COVID,COVID/images/COVID-2604.png,COVID/masks/COVID-2604.png,True,G01743,test,0a7cee93b663d75b2c6e239b99558f33,9469e39612596d6d
COVID-2605.png,COVID,COVID/images/COVID-2605.png,COVID/masks/COVID-2605.png,True,G01744,train,fba7596edd43ce0a3a0324851010a248,d438c35f58c34f58
COVID-2606.png,COVID,COVID/images/COVID-2606.png,COVID/masks/COVID-2606.png,True,G01745,train,529b658aed22be7c92826ad743a1bec7,912ad7425f516b59
COVID-2607.png,COVID,COVID/images/COVID-2607.png,COVID/masks/COVID-2607.png,True,G01746,val,d7029c3af0f66630e4e7aed5d5cac2bf,dd2ad35d5c421f0a
COVID-2608.png,COVID,COVID/images/COVID-2608.png,COVID/masks/COVID-2608.png,True,G01747,test,619db41ad3adc4c61b7a2958b4792d8e,fd28c29795c39b48
COVID-2609.png,COVID,COVID/images/COVID-2609.png,COVID/masks/COVID-2609.png,True,G01748,train,2c1043ce3fc84212a747d923aab6440d,892f9659455a5756
COVID-261.png,COVID,COVID/images/COVID-261.png,COVID/masks/COVID-261.png,True,G01749,val,624c5f3bfb947009e2028a66e2a86ad8,9c4c031f3a6d3c7c
COVID-2610.png,COVID,COVID/images/COVID-2610.png,COVID/masks/COVID-2610.png,True,G01750,train,01019916c069a4e6056146c91d9568e4,d13b9616181c5f5e
COVID-2611.png,COVID,COVID/images/COVID-2611.png,COVID/masks/COVID-2611.png,True,G01751,train,5ad6bc578831be54bb8099a7d2a5c037,cc62416cea2db6bc
COVID-2612.png,COVID,COVID/images/COVID-2612.png,COVID/masks/COVID-2612.png,True,G01752,val,24dd00b55484885173d034ceaa49446c,8568a7875165c7f2
COVID-2613.png,COVID,COVID/images/COVID-2613.png,COVID/masks/COVID-2613.png,True,G01753,train,04b1c83fa6f79db846bf3924932410d9,c6bdd84841d59b8b
COVID-2614.png,COVID,COVID/images/COVID-2614.png,COVID/masks/COVID-2614.png,True,G01754,test,16c237234d647c7226fde4da9bffc32c,997f461e1a494f1a
COVID-2615.png,COVID,COVID/images/COVID-2615.png,COVID/masks/COVID-2615.png,True,G01755,val,58d157b4d91ab18259aed10dce7ed8d4,d43891d34b436b5b
COVID-2616.png,COVID,COVID/images/COVID-2616.png,COVID/masks/COVID-2616.png,True,G01756,train,0e47776fc641c3afdd96f531886ea912,d56643726a686c6e
COVID-2617.png,COVID,COVID/images/COVID-2617.png,COVID/masks/COVID-2617.png,True,G01757,val,b915de41516604e2ef1b61221ecab067,991e56594caaabb8
COVID-2618.png,COVID,COVID/images/COVID-2618.png,COVID/masks/COVID-2618.png,True,G01758,train,8d1b4062cc1e6152c51fec99d1d12d0b,906be767cc1cc84b
COVID-2619.png,COVID,COVID/images/COVID-2619.png,COVID/masks/COVID-2619.png,True,G01759,train,555eb93e4f2383c3ea9d13a6fb155cee,d929b6c63e1e9452
COVID-262.png,COVID,COVID/images/COVID-262.png,COVID/masks/COVID-262.png,True,G01760,train,1bb9fa6c6262f64ed0b1137e66fd706a,d42d50514b4fd74e
COVID-2620.png,COVID,COVID/images/COVID-2620.png,COVID/masks/COVID-2620.png,True,G01761,test,c806bbf9bdb17ba092338c59e1c3410d,d9b3c61cc40f3969
COVID-2621.png,COVID,COVID/images/COVID-2621.png,COVID/masks/COVID-2621.png,True,G01762,train,08db2649a7f01435d2c8d993fabb5026,dd2f961b0d595854
COVID-2622.png,COVID,COVID/images/COVID-2622.png,COVID/masks/COVID-2622.png,True,G01763,train,68789b418347e1dda3241efa8ab49d10,d93b961a5d097a2a
COVID-2623.png,COVID,COVID/images/COVID-2623.png,COVID/masks/COVID-2623.png,True,G01764,train,756e6cc16b7f08c4a8debd3acd6bb47f,d03bf7da58505958
COVID-2624.png,COVID,COVID/images/COVID-2624.png,COVID/masks/COVID-2624.png,True,G01765,test,c04478f490abc30be3c626ed304ee1e1,dc39d34f4a480b3b
COVID-2625.png,COVID,COVID/images/COVID-2625.png,COVID/masks/COVID-2625.png,True,G01766,train,021baf6fdab2b601d4ee1991d7ba19a7,897f041a5945c7fa
COVID-2626.png,COVID,COVID/images/COVID-2626.png,COVID/masks/COVID-2626.png,True,G01767,train,279dfb71eeca0b5620ed8ee896d13461,d475430f4a2e1a6f
COVID-2627.png,COVID,COVID/images/COVID-2627.png,COVID/masks/COVID-2627.png,True,G01768,train,4ced73b24faec0528b89c7fbb627191c,dc3d430b4f2d0a2f
COVID-2628.png,COVID,COVID/images/COVID-2628.png,COVID/masks/COVID-2628.png,True,G01769,train,3ccf56e0389b973e4fbc884131757be6,d92e421b5f461b5c
COVID-2629.png,COVID,COVID/images/COVID-2629.png,COVID/masks/COVID-2629.png,True,G01770,train,223b7070067a528d474fe7a7f70eb947,dd4e4759144b1f4a
COVID-263.png,COVID,COVID/images/COVID-263.png,COVID/masks/COVID-263.png,True,G00465,train,a367a35648fbc180d21f95883abf6dfb,d93f561b444b194e
COVID-2630.png,COVID,COVID/images/COVID-2630.png,COVID/masks/COVID-2630.png,True,G01771,train,5ca6cc3ac34e5a45c359ea23f0e84e99,d52a7e0a4a6b61e3
COVID-2631.png,COVID,COVID/images/COVID-2631.png,COVID/masks/COVID-2631.png,True,G01772,train,483c6e2733a6ecae9f9d9dce7fa3a557,946ae739396436d2
COVID-2632.png,COVID,COVID/images/COVID-2632.png,COVID/masks/COVID-2632.png,True,G01773,train,878dce327a7e48b2f63f6b72bec6fa25,8c2ec3cb4d4e2a6d
COVID-2633.png,COVID,COVID/images/COVID-2633.png,COVID/masks/COVID-2633.png,True,G01774,train,bf1ca50bd35c917666f0a0544601ffe3,d54ac30d4cc1fdd1
COVID-2634.png,COVID,COVID/images/COVID-2634.png,COVID/masks/COVID-2634.png,True,G01775,train,d9d1a180ab8f0db5d00e541e576e410d,886eaec3cb094d57
COVID-2635.png,COVID,COVID/images/COVID-2635.png,COVID/masks/COVID-2635.png,True,G01776,train,ab2a54b5dcfd5259fbba64d4280c0bed,dc2fc34f4c2a2c2e
COVID-2636.png,COVID,COVID/images/COVID-2636.png,COVID/masks/COVID-2636.png,True,G01777,train,13cc27cf58a7b1971f87b4fc4c20248c,9919861a196d79f9
COVID-2637.png,COVID,COVID/images/COVID-2637.png,COVID/masks/COVID-2637.png,True,G01778,train,92714a4444f375beced28d3ef6f636d0,d43c4f59444e3a2f
COVID-2638.png,COVID,COVID/images/COVID-2638.png,COVID/masks/COVID-2638.png,True,G01779,train,050c0eda6117ef26f671ec6cd5b07113,c475d24b2c4a2fce
COVID-2639.png,COVID,COVID/images/COVID-2639.png,COVID/masks/COVID-2639.png,True,G01780,val,f07b4a8e731f7f802d16eebfdb1040be,897b45461697475e
COVID-264.png,COVID,COVID/images/COVID-264.png,COVID/masks/COVID-264.png,True,G01781,val,88d193e3999aa1e56445d18e6a8dd7a5,d47e824b5b464b1b
COVID-2640.png,COVID,COVID/images/COVID-2640.png,COVID/masks/COVID-2640.png,True,G01782,test,3353825bf28ec48885f1f8de5ad554ef,816a860e1a5f5f3b
COVID-2641.png,COVID,COVID/images/COVID-2641.png,COVID/masks/COVID-2641.png,True,G01783,train,25e85213424a429456e718ccd7a0c88a,817a461a594d7a7d
COVID-2642.png,COVID,COVID/images/COVID-2642.png,COVID/masks/COVID-2642.png,True,G01784,test,71e251624af2c995e55a6636da2fbb5b,c13bd61658495d1f
COVID-2643.png,COVID,COVID/images/COVID-2643.png,COVID/masks/COVID-2643.png,True,G01785,train,c8ead6dd4bce3d65a4448fee4a16e6af,c42f031f0d4d5e5e
COVID-2644.png,COVID,COVID/images/COVID-2644.png,COVID/masks/COVID-2644.png,True,G01786,train,d8c088b9fe733784cb847bea7d879889,c43cc3931b0d475f
COVID-2645.png,COVID,COVID/images/COVID-2645.png,COVID/masks/COVID-2645.png,True,G01787,val,1be85d8ae136099047096801cfba8560,c463414c6e6aeeec
COVID-2646.png,COVID,COVID/images/COVID-2646.png,COVID/masks/COVID-2646.png,True,G01788,test,bffb4f676ed585580d72d34f4ae4a4ca,895f06161a595b5f
COVID-2647.png,COVID,COVID/images/COVID-2647.png,COVID/masks/COVID-2647.png,True,G01789,train,4d33596984c96c5790a614b833aafde6,981f861eb9a1a7a5
COVID-2648.png,COVID,COVID/images/COVID-2648.png,COVID/masks/COVID-2648.png,True,G01790,train,e408d443e6d69dadc3c1ccf533926b4b,d909c71e1c551a5f
COVID-2649.png,COVID,COVID/images/COVID-2649.png,COVID/masks/COVID-2649.png,True,G01791,train,eeca9d9c1954890c7296607d39391b8f,837e147b7b7e2028
COVID-265.png,COVID,COVID/images/COVID-265.png,COVID/masks/COVID-265.png,True,G01792,train,1662173a7b6d276601569097b41ed169,817b7f079414565e
COVID-2650.png,COVID,COVID/images/COVID-2650.png,COVID/masks/COVID-2650.png,True,G01793,train,22f1cb4fcda642e442f1cdc004bc5e04,d42ec34bc9c3471b
COVID-2651.png,COVID,COVID/images/COVID-2651.png,COVID/masks/COVID-2651.png,True,G01794,train,0ad25221c746216c32cbdb962bae5ad3,8179c65e1b474e59
COVID-2652.png,COVID,COVID/images/COVID-2652.png,COVID/masks/COVID-2652.png,True,G01795,train,15b134e3aa246472c5b109f844cd2c98,c92d861a1c5d5a7d
COVID-2653.png,COVID,COVID/images/COVID-2653.png,COVID/masks/COVID-2653.png,True,G01796,train,ec41cbf334ad15947a02a9460c1be252,dc0c434b0d2cecbf
COVID-2654.png,COVID,COVID/images/COVID-2654.png,COVID/masks/COVID-2654.png,True,G01797,train,d6ad2a2551b3e9f75d5d1696646ff329,d43f83494b4f3c0e
COVID-2655.png,COVID,COVID/images/COVID-2655.png,COVID/masks/COVID-2655.png,True,G01798,train,8df951a20a19f1862e7134ff79ad2082,d933d61ed41a1a3a
COVID-2656.png,COVID,COVID/images/COVID-2656.png,COVID/masks/COVID-2656.png,True,G01799,train,7d4a1592f4b508828feaf04637bf1a10,806b869e4f977e88
COVID-2657.png,COVID,COVID/images/COVID-2657.png,COVID/masks/COVID-2657.png,True,G01800,val,4921d11114c714a4f0cb338dccecd737,dc6bc39b49411e69
COVID-2658.png,COVID,COVID/images/COVID-2658.png,COVID/masks/COVID-2658.png,True,G01801,train,418d426c1e50d2ca1c0699d2fbdd77d1,807b8714121f7f5b
COVID-2659.png,COVID,COVID/images/COVID-2659.png,COVID/masks/COVID-2659.png,True,G01802,train,bcadf9b80ba1d375c76851d0d4d7c62a,c867131d6f38643e
COVID-266.png,COVID,COVID/images/COVID-266.png,COVID/masks/COVID-266.png,True,G01803,train,3f88e85a14feb1826475be2ef0a7aeb9,d93a161a1c4f1b3d
COVID-2660.png,COVID,COVID/images/COVID-2660.png,COVID/masks/COVID-2660.png,True,G01804,train,60614f6ca31ca928fffb09f88f514d77,c97986161a4b3f3c
COVID-2661.png,COVID,COVID/images/COVID-2661.png,COVID/masks/COVID-2661.png,True,G01805,val,c6d1f0af297860828f44464723c2fe79,c0e92fc62f8a3d0b
COVID-2662.png,COVID,COVID/images/COVID-2662.png,COVID/masks/COVID-2662.png,True,G01806,test,0c2c2526e356d583540d1797bb759a5d,dd29d2c31e54d61a
COVID-2663.png,COVID,COVID/images/COVID-2663.png,COVID/masks/COVID-2663.png,True,G01807,test,c1b44f193125b18a19b8d31841f5a7dc,9139c75f1868393d
COVID-2664.png,COVID,COVID/images/COVID-2664.png,COVID/masks/COVID-2664.png,True,G01808,train,2090411e078f7f4864bf62bdce17f7c3,d47e430b0c6e2c7e
COVID-2665.png,COVID,COVID/images/COVID-2665.png,COVID/masks/COVID-2665.png,True,G01809,test,dad2af8711cf47b665e94824efb6dd5e,993fc71712431e1e
COVID-2666.png,COVID,COVID/images/COVID-2666.png,COVID/masks/COVID-2666.png,True,G01810,train,3e9cd7fd6af03cb9f0760f3f5452240b,d03d4f19586e7cb0
COVID-2667.png,COVID,COVID/images/COVID-2667.png,COVID/masks/COVID-2667.png,True,G01810,train,3e9cd7fd6af03cb9f0760f3f5452240b,d03d4f19586e7cb0
COVID-2668.png,COVID,COVID/images/COVID-2668.png,COVID/masks/COVID-2668.png,True,G01811,train,124130654115aaf4c4518c25de75ccbb,9c6cc35b1d4d1a1d
COVID-2669.png,COVID,COVID/images/COVID-2669.png,COVID/masks/COVID-2669.png,True,G01812,train,6c948e5bcd44f719221f216aa566579d,d92946861aed699d
COVID-267.png,COVID,COVID/images/COVID-267.png,COVID/masks/COVID-267.png,True,G01813,train,9ffb5c65984a2282c58e83e577c3148e,dd4ed2c34f4e0a0b
COVID-2670.png,COVID,COVID/images/COVID-2670.png,COVID/masks/COVID-2670.png,True,G01814,train,04e683f7923643d4a4fd3695e5d993aa,d737d1480c522e6f
COVID-2671.png,COVID,COVID/images/COVID-2671.png,COVID/masks/COVID-2671.png,True,G01815,train,7a41038b3b51da84c5775f06005d0ddb,dc2ec34d4c4169ee
COVID-2672.png,COVID,COVID/images/COVID-2672.png,COVID/masks/COVID-2672.png,True,G01816,train,e20f2d3c27564418a11a7108818b55d3,8772749578937293
COVID-2673.png,COVID,COVID/images/COVID-2673.png,COVID/masks/COVID-2673.png,True,G01817,train,690abe1a88f8747a8adc7cfe7d2995e8,80b97f841e905b5f
COVID-2674.png,COVID,COVID/images/COVID-2674.png,COVID/masks/COVID-2674.png,True,G01818,train,f30711d0160c6e3f2bda303106e332c1,996f871e091f3c25
COVID-2675.png,COVID,COVID/images/COVID-2675.png,COVID/masks/COVID-2675.png,True,G01819,val,e7611f3de6c179d3ec0fa2d5c3b28c09,d93d860317c31b9e
COVID-2676.png,COVID,COVID/images/COVID-2676.png,COVID/masks/COVID-2676.png,True,G01820,train,82f2f2c359f2f11a17d040e47e0a1ac9,816866927a877f35
COVID-2677.png,COVID,COVID/images/COVID-2677.png,COVID/masks/COVID-2677.png,True,G01821,val,eacc92e9c5c3dcddea6daef8ddf76837,dc26470e2635f4d3
COVID-2678.png,COVID,COVID/images/COVID-2678.png,COVID/masks/COVID-2678.png,True,G01821,val,eacc92e9c5c3dcddea6daef8ddf76837,dc26470e2635f4d3
COVID-2679.png,COVID,COVID/images/COVID-2679.png,COVID/masks/COVID-2679.png,True,G01822,train,266452b6feb51542a8a9ea7cc0fe4e24,9960464b1b476d7b
COVID-268.png,COVID,COVID/images/COVID-268.png,COVID/masks/COVID-268.png,True,G01823,test,32c7fd4defc7f6f8cf9c03baf9a8bdc3,817bc6561a596979
COVID-2680.png,COVID,COVID/images/COVID-2680.png,COVID/masks/COVID-2680.png,True,G01824,train,a2a46f31e8c816a39b9fa4e98c992f93,dd2c52d3534f0f11
COVID-2681.png,COVID,COVID/images/COVID-2681.png,COVID/masks/COVID-2681.png,True,G01825,test,3d74749089274c55f9ef70d1c73df43e,d47d53494b4a3a4b
COVID-2682.png,COVID,COVID/images/COVID-2682.png,COVID/masks/COVID-2682.png,True,G01826,val,16eeb28cf3af938968603b7d2fe2d6a8,89e8065a1e1d793f
COVID-2683.png,COVID,COVID/images/COVID-2683.png,COVID/masks/COVID-2683.png,True,G01827,train,0755ad7a56b2b2295f362a94bdb49795,9129d6001e577f1f
COVID-2684.png,COVID,COVID/images/COVID-2684.png,COVID/masks/COVID-2684.png,True,G01828,val,d38a5d228c8521db9121f8194cb0d476,9c2ac71d451e0e6f
COVID-2685.png,COVID,COVID/images/COVID-2685.png,COVID/masks/COVID-2685.png,True,G01829,train,41005f9f4adb5dcb7a16fcdf44a94724,813b7fa4861e5f44
COVID-2686.png,COVID,COVID/images/COVID-2686.png,COVID/masks/COVID-2686.png,True,G01830,test,7820fa931a45329cacdc9d58c0aa2b4e,817b061e5f0e5e59
COVID-2687.png,COVID,COVID/images/COVID-2687.png,COVID/masks/COVID-2687.png,True,G01831,train,d6216b03fb1a630417505ffdf0f13191,880a825bf9f5f5a4
COVID-2688.png,COVID,COVID/images/COVID-2688.png,COVID/masks/COVID-2688.png,True,G01832,train,62cc337419b80d5b83f93ede0eadd530,c9591209aca9f7f1
COVID-2689.png,COVID,COVID/images/COVID-2689.png,COVID/masks/COVID-2689.png,True,G01833,train,96867f8cdfcb715a01a81429c7d94e5f,992dc71e14531f59
COVID-269.png,COVID,COVID/images/COVID-269.png,COVID/masks/COVID-269.png,True,G01834,train,f46de0bf856a82f721c9c78f6c25325c,817f0656594d3c7c
COVID-2690.png,COVID,COVID/images/COVID-2690.png,COVID/masks/COVID-2690.png,True,G01835,train,93ef6a6e887deca9bc9ed43d79daef57,91698e161e5c787e
COVID-2691.png,COVID,COVID/images/COVID-2691.png,COVID/masks/COVID-2691.png,True,G01836,train,1c7a1fba266c21c07765dc07c2a9966f,94946ac34b1b9e7a
COVID-2692.png,COVID,COVID/images/COVID-2692.png,COVID/masks/COVID-2692.png,True,G01837,train,b317cf264f9daa631a10f7381ae692bb,dd244309e60bbd4f
COVID-2693.png,COVID,COVID/images/COVID-2693.png,COVID/masks/COVID-2693.png,True,G01838,train,b4a7cde7562d185be66505ba8212bc3f,946703793e786678
COVID-2694.png,COVID,COVID/images/COVID-2694.png,COVID/masks/COVID-2694.png,True,G01839,val,dbebbf1d0a3d542daac02617850101b3,d52b531e184b3e3a
COVID-2695.png,COVID,COVID/images/COVID-2695.png,COVID/masks/COVID-2695.png,True,G01840,train,397040378bf7165e973334e7f73684f8,842e83594d5f5d4d
COVID-2696.png,COVID,COVID/images/COVID-2696.png,COVID/masks/COVID-2696.png,True,G01841,train,d6f9262460bebc9b1aefb82fac93dc19,817b961e18c75f19
COVID-2697.png,COVID,COVID/images/COVID-2697.png,COVID/masks/COVID-2697.png,True,G01842,val,9ba30da09e542f7a0a7e218e547ee2bb,d97bd61e104b1d49
COVID-2698.png,COVID,COVID/images/COVID-2698.png,COVID/masks/COVID-2698.png,True,G01843,train,093a171ab3905296fbc4d68dec88783d,8928e750786de9ea
COVID-2699.png,COVID,COVID/images/COVID-2699.png,COVID/masks/COVID-2699.png,True,G01844,train,4d2e1ed3d084098bc8ed73febd8be1e6,913b3ed26ac16ac5
COVID-27.png,COVID,COVID/images/COVID-27.png,COVID/masks/COVID-27.png,True,G01845,train,97664faf273e0cc5da6d9f5b4801e199,912c465a5f4f3a69
COVID-270.png,COVID,COVID/images/COVID-270.png,COVID/masks/COVID-270.png,True,G00682,val,3c87443502a630e9a5c15a3a22f39724,9b65c41ae60ebf42
COVID-2700.png,COVID,COVID/images/COVID-2700.png,COVID/masks/COVID-2700.png,True,G01846,test,87cd56a40ca1b528fc5a704a9ab20c90,d46cc39b59474719
COVID-2701.png,COVID,COVID/images/COVID-2701.png,COVID/masks/COVID-2701.png,True,G01847,test,cd9d1c9d6d11ef7b17b411e65fbca5e0,9969851e924f1e7a
COVID-2702.png,COVID,COVID/images/COVID-2702.png,COVID/masks/COVID-2702.png,True,G01848,test,f4ce424783783263e69656efcb94f5ca,dd3a530f1a2d2d34
COVID-2703.png,COVID,COVID/images/COVID-2703.png,COVID/masks/COVID-2703.png,True,G01849,train,ed4bd983b2751805e913edd060e81d4b,827f14141e177f1b
COVID-2704.png,COVID,COVID/images/COVID-2704.png,COVID/masks/COVID-2704.png,True,G01850,train,bcebdc5473d8483385eadbfbafa4325f,d90e864a4d6c6cfc
COVID-2705.png,COVID,COVID/images/COVID-2705.png,COVID/masks/COVID-2705.png,True,G01851,train,b7a057b4192ae5b1fd3354b35f757bd6,83791c94726363cf
COVID-2706.png,COVID,COVID/images/COVID-2706.png,COVID/masks/COVID-2706.png,True,G01852,train,7cbf769eca7ec4f8fdadb22a33e2631c,913bd61859533b53
COVID-2707.png,COVID,COVID/images/COVID-2707.png,COVID/masks/COVID-2707.png,True,G01853,train,39ca146291322e9a1decb57d308dde2d,d1b166c23ed62ec2
COVID-2708.png,COVID,COVID/images/COVID-2708.png,COVID/masks/COVID-2708.png,True,G01854,train,534dd64dc5e56610ac8585e40d51b27f,896d4ec41e901f5f
COVID-2709.png,COVID,COVID/images/COVID-2709.png,COVID/masks/COVID-2709.png,True,G01855,train,5f97c451763666e93934a1374aa28079,893dc7c6166b129d
COVID-271.png,COVID,COVID/images/COVID-271.png,COVID/masks/COVID-271.png,True,G01856,val,659dca501b27549850cdca0fb34ee164,d42c4053dbd3eb43
COVID-2710.png,COVID,COVID/images/COVID-2710.png,COVID/masks/COVID-2710.png,True,G01857,train,cd70d93ed66e8b6e8f27f0211fbe1cd7,817a07185f4c7b6b
COVID-2711.png,COVID,COVID/images/COVID-2711.png,COVID/masks/COVID-2711.png,True,G01858,val,6941b4bb7f65124c50dc656ba233d600,dd2ef2931dd1c00d
COVID-2712.png,COVID,COVID/images/COVID-2712.png,COVID/masks/COVID-2712.png,True,G01859,train,05857c9a10e971f75362411527998902,808f59bc759466da
COVID-2713.png,COVID,COVID/images/COVID-2713.png,COVID/masks/COVID-2713.png,True,G01860,train,d019c1905145cc2ea9f5eee79c3c04d6,957807571b491d3d
COVID-2714.png,COVID,COVID/images/COVID-2714.png,COVID/masks/COVID-2714.png,True,G01861,val,d9220dac5068ac10b38dc03d3700231e,994f861e597d4872
COVID-2715.png,COVID,COVID/images/COVID-2715.png,COVID/masks/COVID-2715.png,True,G01862,test,e27ad0c30ae29e488fa6e57ff63a4a1e,81765c786b2d2e2b
COVID-2716.png,COVID,COVID/images/COVID-2716.png,COVID/masks/COVID-2716.png,True,G01862,test,e27ad0c30ae29e488fa6e57ff63a4a1e,81765c786b2d2e2b
COVID-2717.png,COVID,COVID/images/COVID-2717.png,COVID/masks/COVID-2717.png,True,G01862,test,e27ad0c30ae29e488fa6e57ff63a4a1e,81765c786b2d2e2b
COVID-2718.png,COVID,COVID/images/COVID-2718.png,COVID/masks/COVID-2718.png,True,G01862,test,e27ad0c30ae29e488fa6e57ff63a4a1e,81765c786b2d2e2b
COVID-2719.png,COVID,COVID/images/COVID-2719.png,COVID/masks/COVID-2719.png,True,G01862,test,e27ad0c30ae29e488fa6e57ff63a4a1e,81765c786b2d2e2b
COVID-272.png,COVID,COVID/images/COVID-272.png,COVID/masks/COVID-272.png,True,G01863,train,cb780a9e10ba9a2c4bd9467aadc11954,d13a14041fbdbb33
COVID-2720.png,COVID,COVID/images/COVID-2720.png,COVID/masks/COVID-2720.png,True,G01864,train,ca3d199cfbb46f0914c5d872ffb94fbd,996b849e1e871e17
COVID-2721.png,COVID,COVID/images/COVID-2721.png,COVID/masks/COVID-2721.png,True,G01865,train,38b7ce2230cbd2f31a0868126e406767,856f47060f4b0f5d
COVID-2722.png,COVID,COVID/images/COVID-2722.png,COVID/masks/COVID-2722.png,True,G01866,train,fe4939d6ebc7f52232e7c04af729f64a,91784e47181d7b3b
COVID-2723.png,COVID,COVID/images/COVID-2723.png,COVID/masks/COVID-2723.png,True,G01867,test,cc45b38d3d7a285fe973c28732e80dc2,9969c71e044b397d
COVID-2724.png,COVID,COVID/images/COVID-2724.png,COVID/masks/COVID-2724.png,True,G01868,test,e0639a032b5d8908998df3cec5dbdd1a,942dc3471d473d3c
COVID-2725.png,COVID,COVID/images/COVID-2725.png,COVID/masks/COVID-2725.png,True,G01869,train,43f221f44ffa14f3320db6bf523c5485,d46b831f5d0c3c65
COVID-2726.png,COVID,COVID/images/COVID-2726.png,COVID/masks/COVID-2726.png,True,G01870,train,9000f7da82b2e3cf5c0488b7ecd90ab5,886d071a195d7df8
COVID-2727.png,COVID,COVID/images/COVID-2727.png,COVID/masks/COVID-2727.png,True,G01871,train,c43c2efa0a29253c0f8b12410029f426,89fbd710865ac749
COVID-2728.png,COVID,COVID/images/COVID-2728.png,COVID/masks/COVID-2728.png,True,G01872,train,ebf5188040a0d10e6d5570df0175610c,9c2d870c0d5d5dc7
COVID-2729.png,COVID,COVID/images/COVID-2729.png,COVID/masks/COVID-2729.png,True,G01873,train,5a03960ec19df55c1b2a58325a4706f5,91581e7a79e3a2a5
COVID-273.png,COVID,COVID/images/COVID-273.png,COVID/masks/COVID-273.png,True,G01874,train,4997338a8b7ff5d044abd6754396176c,987f461a6538652f
COVID-2730.png,COVID,COVID/images/COVID-2730.png,COVID/masks/COVID-2730.png,True,G01875,train,7893d2c82bdf57491904f3365a0322a7,817856161b1ff81f
COVID-2731.png,COVID,COVID/images/COVID-2731.png,COVID/masks/COVID-2731.png,True,G01876,train,d0f6fbb9048b09fa3726e9223b97b98c,817b161a197d5b39
COVID-2732.png,COVID,COVID/images/COVID-2732.png,COVID/masks/COVID-2732.png,True,G01877,train,99d3be6d32feaf8e516af82aa35c2917,917b561a193b1b39
COVID-2733.png,COVID,COVID/images/COVID-2733.png,COVID/masks/COVID-2733.png,True,G01878,train,f7c38a1adb64aeb32c5497babf90f808,d13bc606581bdb53
COVID-2734.png,COVID,COVID/images/COVID-2734.png,COVID/masks/COVID-2734.png,True,G01879,train,4e1eaf723762dfbfcf893bd7e2e9545f,d02947425e586b7f
COVID-2735.png,COVID,COVID/images/COVID-2735.png,COVID/masks/COVID-2735.png,True,G01880,train,b7c7357ce2bee856cfff91f30103e5fb,d42cc34b4f4dc92d
COVID-2736.png,COVID,COVID/images/COVID-2736.png,COVID/masks/COVID-2736.png,True,G01881,train,39dbc66c52b579fc499b6b496219cd72,8179079e5e0f3a1b
COVID-2737.png,COVID,COVID/images/COVID-2737.png,COVID/masks/COVID-2737.png,True,G01882,train,3f995bea01926b7d5f105f2205c68be8,a86f471a55653175
COVID-2738.png,COVID,COVID/images/COVID-2738.png,COVID/masks/COVID-2738.png,True,G01883,train,e21d9267bc2059132481abc85d28b0e6,887d47167d473139
COVID-2739.png,COVID,COVID/images/COVID-2739.png,COVID/masks/COVID-2739.png,True,G01884,val,48e07c93f36bb8860300cc287018bddb,c87cc283194f1d5f
COVID-274.png,COVID,COVID/images/COVID-274.png,COVID/masks/COVID-274.png,True,G01885,val,885506ad78f0d04271edd65081b68ef8,807f07061f5d4d67
COVID-2740.png,COVID,COVID/images/COVID-2740.png,COVID/masks/COVID-2740.png,True,G01886,val,a2766b01dfd2fe11e897fe687ba15ad1,d42e53434b4f4e4e
COVID-2741.png,COVID,COVID/images/COVID-2741.png,COVID/masks/COVID-2741.png,True,G01887,test,95596ced1ec610da36fc9dd5faa153d3,81781e595e7a3978
COVID-2742.png,COVID,COVID/images/COVID-2742.png,COVID/masks/COVID-2742.png,True,G01888,test,369afc4ff7b1df00e9830196a64bbd5a,cd16d31f581c4e1b
COVID-2743.png,COVID,COVID/images/COVID-2743.png,COVID/masks/COVID-2743.png,True,G01889,test,e9189c40c6608f884ba64f383a0110c8,d1c92e963a79c133
COVID-2744.png,COVID,COVID/images/COVID-2744.png,COVID/masks/COVID-2744.png,True,G01890,train,b344f68e7fcf1a60abcf30afc62f2398,817956561e581f1f
COVID-2745.png,COVID,COVID/images/COVID-2745.png,COVID/masks/COVID-2745.png,True,G01891,test,e18e47e655c25963a1302bda6f36078b,807d26f6075f8e16
COVID-2746.png,COVID,COVID/images/COVID-2746.png,COVID/masks/COVID-2746.png,True,G01892,train,63e90304944933a2028a58f68d620a88,8979169e5a397929
COVID-2747.png,COVID,COVID/images/COVID-2747.png,COVID/masks/COVID-2747.png,True,G01893,train,afaa02b18e493af58912ef1d397743b2,83688c371a7b3b78
COVID-2748.png,COVID,COVID/images/COVID-2748.png,COVID/masks/COVID-2748.png,True,G01894,train,92dfe512f85bbaee7490b1717a77df7d,c0ec3f933bc63a18
COVID-2749.png,COVID,COVID/images/COVID-2749.png,COVID/masks/COVID-2749.png,True,G01895,val,576adb94dd87dfbe8feb12bd2072b078,fd69d3c217564254
COVID-275.png,COVID,COVID/images/COVID-275.png,COVID/masks/COVID-275.png,True,G01896,train,ba9b6406c05b44ff0d55557f8c3965ef,d92f861e190d3c1f
COVID-2750.png,COVID,COVID/images/COVID-2750.png,COVID/masks/COVID-2750.png,True,G01897,train,d1fd4965c29151bd6b077895abe40c4f,de2ee14a07468f1e
COVID-2751.png,COVID,COVID/images/COVID-2751.png,COVID/masks/COVID-2751.png,True,G01898,train,89677af77e4c8c54d838d3a1d02e6327,817807061e5fee5b
COVID-2752.png,COVID,COVID/images/COVID-2752.png,COVID/masks/COVID-2752.png,True,G01899,train,a89b34af1fda9fa133340a80fd1ded91,d43d82930bc32f9b
COVID-2753.png,COVID,COVID/images/COVID-2753.png,COVID/masks/COVID-2753.png,True,G01900,train,3efd3b2fb3c2e0fded77e5303a8d3109,d42a534b4f456a7c
COVID-2754.png,COVID,COVID/images/COVID-2754.png,COVID/masks/COVID-2754.png,True,G01901,val,c1f06185a975a43786f69176b0ddf056,d13bd61e5e4b2b08
COVID-2755.png,COVID,COVID/images/COVID-2755.png,COVID/masks/COVID-2755.png,True,G01902,train,d83ac6618cde9bbc78194311b04bd1f0,d0bd3bd03bd22bc0
COVID-2756.png,COVID,COVID/images/COVID-2756.png,COVID/masks/COVID-2756.png,True,G01903,train,ee49e962a792dc5f3e8695d5eb3bfe8e,d07dd75259464b58
COVID-2757.png,COVID,COVID/images/COVID-2757.png,COVID/masks/COVID-2757.png,True,G01904,val,2e0b1f2b79c6b29159757900767676f8,817b86561a555e5e
COVID-2758.png,COVID,COVID/images/COVID-2758.png,COVID/masks/COVID-2758.png,True,G01905,train,00e5a0e242e1e1f29e03c936bd8c49a9,817966947e847f91
COVID-2759.png,COVID,COVID/images/COVID-2759.png,COVID/masks/COVID-2759.png,True,G01906,train,519a31e18c0ab1cc0e80b9029cc27b21,917a845259575a5f
COVID-276.png,COVID,COVID/images/COVID-276.png,COVID/masks/COVID-276.png,True,G01907,test,115aa30450a558aa6af42be5ec88a07a,947f43471e194d4d
COVID-2760.png,COVID,COVID/images/COVID-2760.png,COVID/masks/COVID-2760.png,True,G01908,test,f45ba3d2c4ff471dafffc0a63e4ef9b8,887d07165a4f5f58
COVID-2761.png,COVID,COVID/images/COVID-2761.png,COVID/masks/COVID-2761.png,True,G01909,train,ffe01713df8ed79428426f4caffdcfe8,992e06595d796971
COVID-2762.png,COVID,COVID/images/COVID-2762.png,COVID/masks/COVID-2762.png,True,G01910,train,68632462f09572d80e4c0762f3e13f7b,952ec6cf09456f0b
COVID-2763.png,COVID,COVID/images/COVID-2763.png,COVID/masks/COVID-2763.png,True,G01911,train,e1f0daa6dd44f1034e74fcf627301fed,d43f53d50b4a250f
COVID-2764.png,COVID,COVID/images/COVID-2764.png,COVID/masks/COVID-2764.png,True,G01912,train,444f5101af1a13442ba516bd7d618e15,8c7c47db1b4b4b18
COVID-2765.png,COVID,COVID/images/COVID-2765.png,COVID/masks/COVID-2765.png,True,G01913,train,2dd5371a6a10772ff15f60b1bb902d5c,d42f555d0a2d2b2e
COVID-2766.png,COVID,COVID/images/COVID-2766.png,COVID/masks/COVID-2766.png,True,G01914,train,554c8136861fcca0546143cd7ff038e7,dd3ad34f5e480e28
COVID-2767.png,COVID,COVID/images/COVID-2767.png,COVID/masks/COVID-2767.png,True,G01915,train,b09feaa4ae074ccf98e88e7c121e18c6,cd2ec34b0b414f4f
COVID-2768.png,COVID,COVID/images/COVID-2768.png,COVID/masks/COVID-2768.png,True,G01916,train,d2b0bf8d673cc57b5dc3f290a7f7b503,dc2ad34b4b4a4f0b
COVID-2769.png,COVID,COVID/images/COVID-2769.png,COVID/masks/COVID-2769.png,True,G01917,train,d3830df8d2da8f71614f8d3ec17bf973,8178561a185d7f79
COVID-277.png,COVID,COVID/images/COVID-277.png,COVID/masks/COVID-277.png,True,G01918,train,1a04569e91352c22e74f888d06aace5f,dc2b430f2c3cb8f1
COVID-2770.png,COVID,COVID/images/COVID-2770.png,COVID/masks/COVID-2770.png,True,G01919,train,fc8164067305eca3360ae2a1e5f6e28d,8179a0d41f5da85f
COVID-2771.png,COVID,COVID/images/COVID-2771.png,COVID/masks/COVID-2771.png,True,G01920,train,97be8ce6307bd6df80d121bf42b01661,c56e065f1e6c6958
COVID-2772.png,COVID,COVID/images/COVID-2772.png,COVID/masks/COVID-2772.png,True,G01921,train,fd4fa24f5f5677dd8c886c5171cc171e,dc2c021b1f4f1b4f
COVID-2773.png,COVID,COVID/images/COVID-2773.png,COVID/masks/COVID-2773.png,True,G01921,train,fd4fa24f5f5677dd8c886c5171cc171e,dc2c021b1f4f1b4f
COVID-2774.png,COVID,COVID/images/COVID-2774.png,COVID/masks/COVID-2774.png,True,G01921,train,fd4fa24f5f5677dd8c886c5171cc171e,dc2c021b1f4f1b4f
COVID-2775.png,COVID,COVID/images/COVID-2775.png,COVID/masks/COVID-2775.png,True,G01922,train,6b8789db43e10f1a9843973fc86ee3ce,817b0c0c5b794de7
COVID-2776.png,COVID,COVID/images/COVID-2776.png,COVID/masks/COVID-2776.png,True,G01923,val,4a60dc2dcc1e9652a4e3398dfcddfcc1,817d06165b594d7d
COVID-2777.png,COVID,COVID/images/COVID-2777.png,COVID/masks/COVID-2777.png,True,G01924,test,3d543f9c73072eb4fca50188972341f8,813b560a1f39ebb1
COVID-2778.png,COVID,COVID/images/COVID-2778.png,COVID/masks/COVID-2778.png,True,G01925,val,4d5b8487d11941919ea17454aa61323b,d42d42571a4e2f3d
COVID-2779.png,COVID,COVID/images/COVID-2779.png,COVID/masks/COVID-2779.png,True,G01926,val,1dca471fafbd9eba878980ab61002247,992d861b451f594f
COVID-278.png,COVID,COVID/images/COVID-278.png,COVID/masks/COVID-278.png,True,G01927,test,ca02dab1b8dfce4d5ef48c7e823fd888,bd6cc21651c51ecb
COVID-2780.png,COVID,COVID/images/COVID-2780.png,COVID/masks/COVID-2780.png,True,G01928,test,debecd2bf6d3cf2141c04da5f5368e4e,8179861e191a5f5f
COVID-2781.png,COVID,COVID/images/COVID-2781.png,COVID/masks/COVID-2781.png,True,G01929,train,9d22e2fa9e568c39efdcbec415a8063f,913d46461e5c6f4b
COVID-2782.png,COVID,COVID/images/COVID-2782.png,COVID/masks/COVID-2782.png,True,G01930,train,ad6a54cddc70e95f7dc17672b705758c,85787e92788768a7
COVID-2783.png,COVID,COVID/images/COVID-2783.png,COVID/masks/COVID-2783.png,True,G01931,train,664c0b3c76e3032adf06175d67864500,cd2e061e4e4e7a6c
COVID-2784.png,COVID,COVID/images/COVID-2784.png,COVID/masks/COVID-2784.png,True,G01932,train,4da83473f50b298e35e1857d2ca6c8d8,887d165a397163a7
COVID-2785.png,COVID,COVID/images/COVID-2785.png,COVID/masks/COVID-2785.png,True,G01933,train,11def4a0c3d13190908cccb567a2ad37,813f06c71b1d4d4f
COVID-2786.png,COVID,COVID/images/COVID-2786.png,COVID/masks/COVID-2786.png,True,G01934,train,74bf8b94dd8f2bfeb7dc696d77e828a0,dc2d434f0e2b2cbc
COVID-2787.png,COVID,COVID/images/COVID-2787.png,COVID/masks/COVID-2787.png,True,G01935,train,4bfebed594deac0cb2f9d8a143226854,917bd61a192f32a6
COVID-2788.png,COVID,COVID/images/COVID-2788.png,COVID/masks/COVID-2788.png,True,G01936,val,65aaefafaf15efbac7d154739299bc36,9322dc94791b567a
COVID-2789.png,COVID,COVID/images/COVID-2789.png,COVID/masks/COVID-2789.png,True,G01937,val,cb3949507937c766b9763879182e6ec4,917856151b9f3995
COVID-279.png,COVID,COVID/images/COVID-279.png,COVID/masks/COVID-279.png,True,G01938,val,a1530f914cc09dada70e7f7c7d1e8a1f,d83166c24eca1fdc
COVID-2790.png,COVID,COVID/images/COVID-2790.png,COVID/masks/COVID-2790.png,True,G01939,test,f7fc0ee0b411dc30c829b8959ab87435,817956d61a1e5b1b
COVID-2791.png,COVID,COVID/images/COVID-2791.png,COVID/masks/COVID-2791.png,True,G01940,train,2a21cc98b73eb869e03f8a8829e9ce81,d43d43410b2f3d3d
COVID-2792.png,COVID,COVID/images/COVID-2792.png,COVID/masks/COVID-2792.png,True,G01941,train,2681ac64bec187f3141a3587bcd94dc5,913b061e1a4f5f3a
COVID-2793.png,COVID,COVID/images/COVID-2793.png,COVID/masks/COVID-2793.png,True,G01942,train,deec7920db8958c17925725651663058,8179861a594e5b5f
COVID-2794.png,COVID,COVID/images/COVID-2794.png,COVID/masks/COVID-2794.png,True,G01943,train,1121e54324e389d13082a749881d0c9a,f979c61b1d481ba4
COVID-2795.png,COVID,COVID/images/COVID-2795.png,COVID/masks/COVID-2795.png,True,G01944,train,7602205feb135502201f416cadc214f5,807f05161f5f1a5e
COVID-2796.png,COVID,COVID/images/COVID-2796.png,COVID/masks/COVID-2796.png,True,G01945,train,1a2a08658698b13a62744a1b728a1922,997b461a18493b7d
COVID-2797.png,COVID,COVID/images/COVID-2797.png,COVID/masks/COVID-2797.png,True,G01946,train,fc51173f70735a0ef045369e72587f80,817d461e105f5d75
COVID-2798.png,COVID,COVID/images/COVID-2798.png,COVID/masks/COVID-2798.png,True,G01947,test,7f29c430800952fd0b37c6d2d929b80b,8979061e5a5979e9
COVID-2799.png,COVID,COVID/images/COVID-2799.png,COVID/masks/COVID-2799.png,True,G01948,test,ee2ee0539249fa739798189253c2e68a,cc2c535f1d1aa92d
COVID-28.png,COVID,COVID/images/COVID-28.png,COVID/masks/COVID-28.png,True,G01949,train,d0e57fdd44c41de1cf84e95d3567ba3d,dd3bd34b14625c1c
COVID-280.png,COVID,COVID/images/COVID-280.png,COVID/masks/COVID-280.png,True,G01918,train,1a04569e91352c22e74f888d06aace5f,dc2b430f2c3cb8f1
COVID-2800.png,COVID,COVID/images/COVID-2800.png,COVID/masks/COVID-2800.png,True,G01950,train,d84d8fd2b905d87114907b26324c7049,d43cb1c30d4be365
COVID-2801.png,COVID,COVID/images/COVID-2801.png,COVID/masks/COVID-2801.png,True,G01951,train,7ac125562f3fa69e46c5d6ed1aefdbab,d4294b2e2e343737
COVID-2802.png,COVID,COVID/images/COVID-2802.png,COVID/masks/COVID-2802.png,True,G01952,train,893d52480c1be7896eb44ae1fd0aa2ef,946c06094f7e69ee
COVID-2803.png,COVID,COVID/images/COVID-2803.png,COVID/masks/COVID-2803.png,True,G01953,val,0cdf7d597903608f2c4cae5f6314c309,d43d434f0b4e6c78
COVID-2804.png,COVID,COVID/images/COVID-2804.png,COVID/masks/COVID-2804.png,True,G01954,train,86c2accca2b44eaa027e34474ab4fc68,987d871e1d4d1a59
COVID-2805.png,COVID,COVID/images/COVID-2805.png,COVID/masks/COVID-2805.png,True,G01954,train,86c2accca2b44eaa027e34474ab4fc68,987d871e1d4d1a59
COVID-2806.png,COVID,COVID/images/COVID-2806.png,COVID/masks/COVID-2806.png,True,G01955,train,f9af3d761249641d4c29c21dcd480927,997d871e17051f19
COVID-2807.png,COVID,COVID/images/COVID-2807.png,COVID/masks/COVID-2807.png,True,G01956,train,88b2849b8322667577a0d7afea9ec953,887d871e175f184b
COVID-2808.png,COVID,COVID/images/COVID-2808.png,COVID/masks/COVID-2808.png,True,G01956,train,88b2849b8322667577a0d7afea9ec953,887d871e175f184b
COVID-2809.png,COVID,COVID/images/COVID-2809.png,COVID/masks/COVID-2809.png,True,G01956,train,c6e5471758bfa2b35fe0d29413e89938,887d871e175f184b
COVID-281.png,COVID,COVID/images/COVID-281.png,COVID/masks/COVID-281.png,True,G01957,train,fc79bc8ed3f5819446a0822ec741bb3a,9c2c43930bcf1ccf
COVID-2810.png,COVID,COVID/images/COVID-2810.png,COVID/masks/COVID-2810.png,True,G01956,train,c6e5471758bfa2b35fe0d29413e89938,887d871e175f184b
COVID-2811.png,COVID,COVID/images/COVID-2811.png,COVID/masks/COVID-2811.png,True,G01956,train,c6e5471758bfa2b35fe0d29413e89938,887d871e175f184b
COVID-2812.png,COVID,COVID/images/COVID-2812.png,COVID/masks/COVID-2812.png,True,G01956,train,c6e5471758bfa2b35fe0d29413e89938,887d871e175f184b
COVID-2813.png,COVID,COVID/images/COVID-2813.png,COVID/masks/COVID-2813.png,True,G01956,train,c6e5471758bfa2b35fe0d29413e89938,887d871e175f184b
COVID-2814.png,COVID,COVID/images/COVID-2814.png,COVID/masks/COVID-2814.png,True,G01958,train,8cf806f80c77a5dee39ebe16c2109641,d07cd5431d4e0e79
COVID-2815.png,COVID,COVID/images/COVID-2815.png,COVID/masks/COVID-2815.png,True,G01959,train,815c82180379460de9f15146b9670e1b,de2dd30b4d0c2d6c
COVID-2816.png,COVID,COVID/images/COVID-2816.png,COVID/masks/COVID-2816.png,True,G01960,train,56aeb49ff422f7042b8fd4765dfd42b3,d43dc2535f424b1e
COVID-2817.png,COVID,COVID/images/COVID-2817.png,COVID/masks/COVID-2817.png,True,G01961,train,b93dc81032d1da5c896ff3d9f2631b1c,c42d424a6de47df4
COVID-2818.png,COVID,COVID/images/COVID-2818.png,COVID/masks/COVID-2818.png,True,G01962,train,d63667c5653bfa08f95d65e204689115,9128061f5f5e7a78
COVID-2819.png,COVID,COVID/images/COVID-2819.png,COVID/masks/COVID-2819.png,True,G01963,train,69875a72f278df2fe41f1a87c89d501e,d0bc03431353cf6f
COVID-282.png,COVID,COVID/images/COVID-282.png,COVID/masks/COVID-282.png,True,G01964,train,22e9022cfc312ab7071d070276174dce,99ff86100a1f394f
COVID-2820.png,COVID,COVID/images/COVID-2820.png,COVID/masks/COVID-2820.png,True,G01965,val,650020ff6fb2c41b50c6dcf6b1a40c97,991b561a19693b3e
COVID-2821.png,COVID,COVID/images/COVID-2821.png,COVID/masks/COVID-2821.png,True,G01966,test,2bad54f5d9f987a6b195cc6d4e2f067e,a87f865d554e5948
COVID-2822.png,COVID,COVID/images/COVID-2822.png,COVID/masks/COVID-2822.png,True,G01967,val,435935c948d0604f08dc00e7f4ec8dbb,897d86d65b085f19
COVID-2823.png,COVID,COVID/images/COVID-2823.png,COVID/masks/COVID-2823.png,True,G01967,val,435935c948d0604f08dc00e7f4ec8dbb,897d86d65b085f19
COVID-2824.png,COVID,COVID/images/COVID-2824.png,COVID/masks/COVID-2824.png,True,G01968,train,0e814a8d26fd0abdd9e711680c48b270,dd0a0b0d2cbdb5b2
COVID-2825.png,COVID,COVID/images/COVID-2825.png,COVID/masks/COVID-2825.png,True,G01969,test,40c96a1d129309a01a09704e178bd9a5,803c57c75f1c4f43
COVID-2826.png,COVID,COVID/images/COVID-2826.png,COVID/masks/COVID-2826.png,True,G01970,test,807008d7facb1431664517448b5667fe,993907561e1c5f5a
COVID-2827.png,COVID,COVID/images/COVID-2827.png,COVID/masks/COVID-2827.png,True,G01971,train,016bc4c87c822cc4990af6d24820398a,f929c64b1d864b1d
COVID-2828.png,COVID,COVID/images/COVID-2828.png,COVID/masks/COVID-2828.png,True,G01972,train,592003d17881edf6bb8ffc3921f93996,917956165a59693b
COVID-2829.png,COVID,COVID/images/COVID-2829.png,COVID/masks/COVID-2829.png,True,G01973,train,d36275b3ace27028124e1c6db160a147,9138161f5a7b7968
COVID-283.png,COVID,COVID/images/COVID-283.png,COVID/masks/COVID-283.png,True,G01974,train,ef505d244f4b465ae177489c8a595b6f,dd2dc29097c56a93
COVID-2830.png,COVID,COVID/images/COVID-2830.png,COVID/masks/COVID-2830.png,True,G01975,val,68c6eddfda4d9c9e812c2c89facfe4f5,d839154b96584bf5
COVID-2831.png,COVID,COVID/images/COVID-2831.png,COVID/masks/COVID-2831.png,True,G01976,train,ae60bd52eb29e25a7209e0962e1af760,883fa7ce1a3d290d
COVID-2832.png,COVID,COVID/images/COVID-2832.png,COVID/masks/COVID-2832.png,True,G01977,train,ea2ec0099bd013d58f146e24986377f9,882f535b747272f0
COVID-2833.png,COVID,COVID/images/COVID-2833.png,COVID/masks/COVID-2833.png,True,G01978,train,1bcfd302a574c1be3dd2dafc99df00be,987f055e185b4ba5
COVID-2834.png,COVID,COVID/images/COVID-2834.png,COVID/masks/COVID-2834.png,True,G01979,train,9c2947e3bd7d5abb36f6ac7656fc6473,817b845e1e9e5a1b
COVID-2835.png,COVID,COVID/images/COVID-2835.png,COVID/masks/COVID-2835.png,True,G01980,test,7e8beb58b6d945bce50619b9b571681d,d66ac1494d4e2f2d
COVID-2836.png,COVID,COVID/images/COVID-2836.png,COVID/masks/COVID-2836.png,True,G01981,test,b4554a210cce6776edc1f36c211e2986,c4c42fc3cf2dcc2c
COVID-2837.png,COVID,COVID/images/COVID-2837.png,COVID/masks/COVID-2837.png,True,G01982,train,3bc6934c265fb832e4f226b147d83939,c03cb743af064fac
COVID-2838.png,COVID,COVID/images/COVID-2838.png,COVID/masks/COVID-2838.png,True,G01983,test,1064c7ce6a3973c1ee44b73c3c886e47,95996a967a69c562
COVID-2839.png,COVID,COVID/images/COVID-2839.png,COVID/masks/COVID-2839.png,True,G01984,test,e45724663f15152cdee1679ac3667dcb,d42d43530c6d6b1f
COVID-284.png,COVID,COVID/images/COVID-284.png,COVID/masks/COVID-284.png,True,G00349,train,763013eef54d3c8da6acb9c519a4fc6e,cd2c1253434b1f6f
COVID-2840.png,COVID,COVID/images/COVID-2840.png,COVID/masks/COVID-2840.png,True,G01985,train,eacadb143126b637870b41175ee04ff8,d42e404b4f6f4c4f
COVID-2841.png,COVID,COVID/images/COVID-2841.png,COVID/masks/COVID-2841.png,True,G01986,train,5543d98361a6168cb83cdae8819382be,88474269746ee6cf
COVID-2842.png,COVID,COVID/images/COVID-2842.png,COVID/masks/COVID-2842.png,True,G01987,train,4a6da5dda0b449dafaca0fef29154aa2,8967075b4f594770
COVID-2843.png,COVID,COVID/images/COVID-2843.png,COVID/masks/COVID-2843.png,True,G01988,test,7b8eb71d1427febfa55e6cf02bec4cd1,dc2ad30f5e0f4a0b
COVID-2844.png,COVID,COVID/images/COVID-2844.png,COVID/masks/COVID-2844.png,True,G01989,train,883cf6b9f610d9ac1a1e700272d83d87,81686e977b857ec0
COVID-2845.png,COVID,COVID/images/COVID-2845.png,COVID/masks/COVID-2845.png,True,G01990,val,8fc39bcb85e6466745ee3acd04316420,d93ec652185b1b39
COVID-2846.png,COVID,COVID/images/COVID-2846.png,COVID/masks/COVID-2846.png,True,G01991,val,2274d6ae4f97981aae8b1fdd62f1dc5e,d42e230b2b3b3f25
COVID-2847.png,COVID,COVID/images/COVID-2847.png,COVID/masks/COVID-2847.png,True,G01992,train,e139c79a347c80dd77bdddfe0aacada2,816badd594ce9515
COVID-2848.png,COVID,COVID/images/COVID-2848.png,COVID/masks/COVID-2848.png,True,G01993,train,7e5058a02fac9ee272cb58f9fc73ac15,893b07561e545b5b
COVID-2849.png,COVID,COVID/images/COVID-2849.png,COVID/masks/COVID-2849.png,True,G01994,train,9849b7d7546914dd5845b90279729006,897d87061ac75f1a
COVID-285.png,COVID,COVID/images/COVID-285.png,COVID/masks/COVID-285.png,True,G01995,train,455208e9d70022c1174bbbf58b329ecb,992c861b1f4f1a5d
COVID-2850.png,COVID,COVID/images/COVID-2850.png,COVID/masks/COVID-2850.png,True,G01996,train,35f04657e6158a3e78e2f83407eb7093,913f46182a7db9e1
COVID-2851.png,COVID,COVID/images/COVID-2851.png,COVID/masks/COVID-2851.png,True,G01997,train,2fc803cbc6d90476f38685b54ec46b72,905d4e796c6cb1e4
COVID-2852.png,COVID,COVID/images/COVID-2852.png,COVID/masks/COVID-2852.png,True,G01998,train,7e2dee9738cf84420344928e97335b45,817f868a57471b1b
COVID-2853.png,COVID,COVID/images/COVID-2853.png,COVID/masks/COVID-2853.png,True,G01999,train,9299153f715d99cadedb4c005b89ae5e,897f861a495d78c3
COVID-2854.png,COVID,COVID/images/COVID-2854.png,COVID/masks/COVID-2854.png,True,G02000,train,dcdc785e02f0a8759a267a3d74f60f07,d1394606be4b4d3d
COVID-2855.png,COVID,COVID/images/COVID-2855.png,COVID/masks/COVID-2855.png,True,G02001,train,4660075a0585405188b06d7019f5e2ab,8c6c875b4d5d4a69
COVID-2856.png,COVID,COVID/images/COVID-2856.png,COVID/masks/COVID-2856.png,True,G02002,val,64de5cf4dfb6ea15ddfba839cedfee85,8c6e835b4d1f4a65
COVID-2857.png,COVID,COVID/images/COVID-2857.png,COVID/masks/COVID-2857.png,True,G02003,train,114b7205b3e11e15643dd00b8eac1d83,d52dd6521c4e4b59
COVID-2858.png,COVID,COVID/images/COVID-2858.png,COVID/masks/COVID-2858.png,True,G02004,train,3741a4580fe183dfaf04171c576699a5,cd2e93530f4e1a59
COVID-2859.png,COVID,COVID/images/COVID-2859.png,COVID/masks/COVID-2859.png,True,G02005,train,1b037283f1b132f92702b91648346928,916d5e5bd61e110e
COVID-286.png,COVID,COVID/images/COVID-286.png,COVID/masks/COVID-286.png,True,G00481,train,6a76420db78261a3c222d8b601bc0908,85780f070e1f5e1f
COVID-2860.png,COVID,COVID/images/COVID-2860.png,COVID/masks/COVID-2860.png,True,G02006,train,2571453f57085796e97842fa0406fe49,81685e10155f7b7e
COVID-2861.png,COVID,COVID/images/COVID-2861.png,COVID/masks/COVID-2861.png,True,G02007,val,75823c02f39b656eabc97d56d2773d98,83798550115f5f3b
COVID-2862.png,COVID,COVID/images/COVID-2862.png,COVID/masks/COVID-2862.png,True,G02008,val,4610a953bf871d3b01bf9236616864a2,817b15161a195f7b
COVID-2863.png,COVID,COVID/images/COVID-2863.png,COVID/masks/COVID-2863.png,True,G02009,test,b4dd5a4f532f45b11ab1d46d138d4870,817b041e1a5f5a7b
COVID-2864.png,COVID,COVID/images/COVID-2864.png,COVID/masks/COVID-2864.png,True,G02010,train,eda19479a756dea6fcf2ec756060c1a0,9169c51db2c55cb9
COVID-2865.png,COVID,COVID/images/COVID-2865.png,COVID/masks/COVID-2865.png,True,G02011,train,b3403503bc648aa6feb435fd2f52fc95,9c35430bc30f5b4f
COVID-2866.png,COVID,COVID/images/COVID-2866.png,COVID/masks/COVID-2866.png,True,G02012,train,206fb13d517cbe26fc3ebad682a05998,d179c6461a5e4e5a
COVID-2867.png,COVID,COVID/images/COVID-2867.png,COVID/masks/COVID-2867.png,True,G02013,train,67d115390965bb0d55b3bf77b499439a,d95946061b0d4d7f
COVID-2868.png,COVID,COVID/images/COVID-2868.png,COVID/masks/COVID-2868.png,True,G02014,train,d4f4df2182b84a4245f96296bc963f1b,9969861c5e1e1a5f
COVID-2869.png,COVID,COVID/images/COVID-2869.png,COVID/masks/COVID-2869.png,True,G02015,train,80a0d818a27b1bce8f34323772810642,887f861e4b1d194f
COVID-287.png,COVID,COVID/images/COVID-287.png,COVID/masks/COVID-287.png,True,G00356,train,9f6a3086e00f0da2b5b998fbe8513a55,9c6f821bc71e194d
COVID-2870.png,COVID,COVID/images/COVID-2870.png,COVID/masks/COVID-2870.png,True,G02016,train,95721b9493c8e4db4a03559fd07f3d57,997f84145b931a5b
COVID-2871.png,COVID,COVID/images/COVID-2871.png,COVID/masks/COVID-2871.png,True,G02017,train,8a1bcb2ba7c1e068aab19cd1d82757d2,dc29075c2879ece6
COVID-2872.png,COVID,COVID/images/COVID-2872.png,COVID/masks/COVID-2872.png,True,G02018,train,c512ad4df5affe82239c77fef4f2587a,d02dc3165e4f195b
COVID-2873.png,COVID,COVID/images/COVID-2873.png,COVID/masks/COVID-2873.png,True,G02019,train,9281d87e0a1eb4b472ac11268f21fc8a,81781e5e5a796879
COVID-2874.png,COVID,COVID/images/COVID-2874.png,COVID/masks/COVID-2874.png,True,G02020,train,3e4c30946fc69cf9199884935e644e14,d42e574b1c6c4a5e
COVID-2875.png,COVID,COVID/images/COVID-2875.png,COVID/masks/COVID-2875.png,True,G02021,val,00e18849a23ffbc34d4df9be513700d1,d42e835b186d6af8
COVID-2876.png,COVID,COVID/images/COVID-2876.png,COVID/masks/COVID-2876.png,True,G02022,train,4a3615c3866b3228c55efbfa994f6457,c90e867aa1e3e1a7
COVID-2877.png,COVID,COVID/images/COVID-2877.png,COVID/masks/COVID-2877.png,True,G02023,train,0356170e1be8468968fd8d8cd2f2703a,d83b861a1f58786e
COVID-2878.png,COVID,COVID/images/COVID-2878.png,COVID/masks/COVID-2878.png,True,G02024,train,5a59c00d659ede40a41b90af037c1f61,856c7f96165a1b49
COVID-2879.png,COVID,COVID/images/COVID-2879.png,COVID/masks/COVID-2879.png,True,G02025,test,c479a603d8956b7d2fe80f43efc6fc02,dc0d430f4c0c3ff6
COVID-288.png,COVID,COVID/images/COVID-288.png,COVID/masks/COVID-288.png,True,G02026,val,70169f269ce5204d6ab21867539c9ffa,9c2f031d4d5c4d79
COVID-2880.png,COVID,COVID/images/COVID-2880.png,COVID/masks/COVID-2880.png,True,G02027,val,d4999f9b164dba68419f72b24f3bf1d0,817f061e59695d78
COVID-2881.png,COVID,COVID/images/COVID-2881.png,COVID/masks/COVID-2881.png,True,G02028,test,e15fea98d4481efe2edbd6f4dd46fbf1,817f065a19495f79
COVID-2882.png,COVID,COVID/images/COVID-2882.png,COVID/masks/COVID-2882.png,True,G02029,test,c33524cdda485b9a739f18d4c175db27,9d6c871b594e7868
COVID-2883.png,COVID,COVID/images/COVID-2883.png,COVID/masks/COVID-2883.png,True,G02030,test,c0f907433a6884732f1b55b39c4d35d6,996c1e596d78e0a6
COVID-2884.png,COVID,COVID/images/COVID-2884.png,COVID/masks/COVID-2884.png,True,G02031,train,51632fb6fc60892c953719dbaba5c2d4,d42e42cb0b4f0f4f
COVID-2885.png,COVID,COVID/images/COVID-2885.png,COVID/masks/COVID-2885.png,True,G02032,train,e45e4977ef276ef3f68a40ff1d2d39f3,d42e434f0b4a2f7c
COVID-2886.png,COVID,COVID/images/COVID-2886.png,COVID/masks/COVID-2886.png,True,G02033,train,4ce282a74d3b56f90ba966094840947e,917887471e4f0f1b
COVID-2887.png,COVID,COVID/images/COVID-2887.png,COVID/masks/COVID-2887.png,True,G02034,train,9f0ca947b180b13df2e32862f24e6ae5,81788656195f1f5b
COVID-2888.png,COVID,COVID/images/COVID-2888.png,COVID/masks/COVID-2888.png,True,G02035,test,3d1337f9101216d580c45323e8c3a5ab,83581c5a5b7978e9
COVID-2889.png,COVID,COVID/images/COVID-2889.png,COVID/masks/COVID-2889.png,True,G02036,train,e90503dd2feeff95dce70f069db48cc6,817956467a5e5a6a
COVID-289.png,COVID,COVID/images/COVID-289.png,COVID/masks/COVID-289.png,True,G02037,train,2bb97ce2208f894771cdfe0f333a8a68,d951d7565a78103b
COVID-2890.png,COVID,COVID/images/COVID-2890.png,COVID/masks/COVID-2890.png,True,G02038,test,031d29d57f271c314de26a991351c4ce,947ecf59554c184e
COVID-2891.png,COVID,COVID/images/COVID-2891.png,COVID/masks/COVID-2891.png,True,G02039,train,acab3800d038152e6e282421c9ef137e,943ee64f19c70f18
COVID-2892.png,COVID,COVID/images/COVID-2892.png,COVID/masks/COVID-2892.png,True,G02040,train,b13bfa8effce8d770c61ed8969d4cb00,9c6c471a096c7cf5
COVID-2893.png,COVID,COVID/images/COVID-2893.png,COVID/masks/COVID-2893.png,True,G02041,train,991b0653fe67b9078fb789b6ec0ddc19,d42e4347594f4e5c
COVID-2894.png,COVID,COVID/images/COVID-2894.png,COVID/masks/COVID-2894.png,True,G02042,train,58bb5973cbb842f3f0e75f65cba43192,dd2cc31b4d461b4d
COVID-2895.png,COVID,COVID/images/COVID-2895.png,COVID/masks/COVID-2895.png,True,G02043,train,0b0f91efebe9de69b91ac51682898042,dd2f421f4e4d4a1c
COVID-2896.png,COVID,COVID/images/COVID-2896.png,COVID/masks/COVID-2896.png,True,G02044,train,ef1c98ae177beb7c4e0ce03976c54666,dd2a470a29b8ece5
COVID-2897.png,COVID,COVID/images/COVID-2897.png,COVID/masks/COVID-2897.png,True,G02045,train,c300cacbdc4d7ae315d016cf0d467322,996d469a2f8939c5
COVID-2898.png,COVID,COVID/images/COVID-2898.png,COVID/masks/COVID-2898.png,True,G02046,train,171a29134b61f63c0a73579905cd8e5a,d42f4b4b4b2c2b2e
COVID-2899.png,COVID,COVID/images/COVID-2899.png,COVID/masks/COVID-2899.png,True,G02047,val,3e96a42848ca990f5ddea523e672b762,8135a72e3c707575
COVID-29.png,COVID,COVID/images/COVID-29.png,COVID/masks/COVID-29.png,True,G01352,test,c6b92de363cd82e4efd585901fba51cb,bd6dc61617451a1d
COVID-290.png,COVID,COVID/images/COVID-290.png,COVID/masks/COVID-290.png,True,G02048,val,40d94339c2a322d0138b34f524e9b259,807fe2171d4d3c3c
COVID-2900.png,COVID,COVID/images/COVID-2900.png,COVID/masks/COVID-2900.png,True,G02049,train,e6b23052d102ef279b3771ef4e3e69ad,897f07161a4f1f1a
COVID-2901.png,COVID,COVID/images/COVID-2901.png,COVID/masks/COVID-2901.png,True,G02050,train,b8ced71c263e7de641617596b44e97bf,916e871b4f061b5b
COVID-2902.png,COVID,COVID/images/COVID-2902.png,COVID/masks/COVID-2902.png,True,G02051,val,7a71a6d7946fd37068a14c8e0956c876,817d87921f87171b
COVID-2903.png,COVID,COVID/images/COVID-2903.png,COVID/masks/COVID-2903.png,True,G02052,train,dc554a6bcd69483436a3d74acf6b0610,d57e030b196c6d3c
COVID-2904.png,COVID,COVID/images/COVID-2904.png,COVID/masks/COVID-2904.png,True,G02053,train,5c45e411bfd95c29ebd3b2dc7c998ec9,812976c61f987bc6
COVID-2905.png,COVID,COVID/images/COVID-2905.png,COVID/masks/COVID-2905.png,True,G02054,train,ea20ab89d1520155c0fde6e033c0ef96,817b07165ad94f78
COVID-2906.png,COVID,COVID/images/COVID-2906.png,COVID/masks/COVID-2906.png,True,G02055,train,f57390cf718828a225d8fe51fdad46d4,913a0e111f7b7a6c
COVID-2907.png,COVID,COVID/images/COVID-2907.png,COVID/masks/COVID-2907.png,True,G02056,train,d4b8feb1e37432089afc28af152d46b0,9979861e1a595d5a
COVID-2908.png,COVID,COVID/images/COVID-2908.png,COVID/masks/COVID-2908.png,True,G02057,train,d1b32e3c7c396a5877b93e3a2ab66e7c,d56ac35d486b086f
COVID-2909.png,COVID,COVID/images/COVID-2909.png,COVID/masks/COVID-2909.png,True,G02058,train,a3f8929307a55e88138c120d3806afca,992f871b0f083caf
COVID-291.png,COVID,COVID/images/COVID-291.png,COVID/masks/COVID-291.png,True,G02059,train,43843fafeaefc69efa5329728a6654b7,fa59c1960fc29e0b
COVID-2910.png,COVID,COVID/images/COVID-2910.png,COVID/masks/COVID-2910.png,True,G02060,train,642956ce78ad9de0f207e36b3594c816,d46ec35d4c2e0a4f
COVID-2911.png,COVID,COVID/images/COVID-2911.png,COVID/masks/COVID-2911.png,True,G02061,train,c66b7ffc88465648d2e65d3d4d34ad7b,817b54184b6f3e3a
COVID-2912.png,COVID,COVID/images/COVID-2912.png,COVID/masks/COVID-2912.png,True,G02062,train,5da278684d3d7c9ede117008ce5b2255,817bb416194f5f19
COVID-2913.png,COVID,COVID/images/COVID-2913.png,COVID/masks/COVID-2913.png,True,G02063,train,7b2b982fa8a3096a2a31f50f05a865ae,8179e79e1c4bc91c
COVID-2914.png,COVID,COVID/images/COVID-2914.png,COVID/masks/COVID-2914.png,True,G02064,train,5d13fe9ed58a48da0d0869781659aa9c,996dc4161a5d4d5e
COVID-2915.png,COVID,COVID/images/COVID-2915.png,COVID/masks/COVID-2915.png,True,G02065,val,1f1d64ae28610ed3f378ab4f9d1ab03b,887b87161e1b5d1d
COVID-2916.png,COVID,COVID/images/COVID-2916.png,COVID/masks/COVID-2916.png,True,G02066,test,bd834f567c39b0002a88c53fdc54405e,cc3f530f24313735
COVID-2917.png,COVID,COVID/images/COVID-2917.png,COVID/masks/COVID-2917.png,True,G02067,test,6bbcb60c1db108d723c66af0cad2ede6,c57f02571e4d1a1b
COVID-2918.png,COVID,COVID/images/COVID-2918.png,COVID/masks/COVID-2918.png,True,G02068,test,0cf289f00f81c257ae946a13e4035ce8,d4bde283c1432de5
COVID-2919.png,COVID,COVID/images/COVID-2919.png,COVID/masks/COVID-2919.png,True,G02069,train,9215a7f5381be7a13020629e01f2a314,d42c43571bdb4347
COVID-292.png,COVID,COVID/images/COVID-292.png,COVID/masks/COVID-292.png,True,G00369,train,bf4f4abe008c51642772d8657af56c54,983dc55657c3461e
COVID-2920.png,COVID,COVID/images/COVID-2920.png,COVID/masks/COVID-2920.png,True,G02070,val,a0c009001b0b8e09a4813220772ec346,817fc614584b5b5b
COVID-2921.png,COVID,COVID/images/COVID-2921.png,COVID/masks/COVID-2921.png,True,G02071,val,8458f08604826380e5191713e7e8b6e2,cd22870f9129f2fc
COVID-2922.png,COVID,COVID/images/COVID-2922.png,COVID/masks/COVID-2922.png,True,G02072,train,7acb257ce3594262ceda17d7cec56848,885f061a595f5aea
COVID-2923.png,COVID,COVID/images/COVID-2923.png,COVID/masks/COVID-2923.png,True,G02073,train,e89964260c06a01a201cca08a5d420c7,817b164e5b794978
COVID-2924.png,COVID,COVID/images/COVID-2924.png,COVID/masks/COVID-2924.png,True,G02074,train,2051a92b3ac0e96721151fbb45825104,8979079e1a495f39
COVID-2925.png,COVID,COVID/images/COVID-2925.png,COVID/masks/COVID-2925.png,True,G02075,train,d6aaa99235d3515bf4d5d38c52a19c6e,947c4f09796ce4d2
COVID-2926.png,COVID,COVID/images/COVID-2926.png,COVID/masks/COVID-2926.png,True,G02076,val,60c52e2eba452a6588db0bad2b4a351f,907f461a5b4f5a0b
COVID-2927.png,COVID,COVID/images/COVID-2927.png,COVID/masks/COVID-2927.png,True,G02077,train,c54d147b7ea1348c326f45d6f16e6039,d02f57084c6b7f0b
COVID-2928.png,COVID,COVID/images/COVID-2928.png,COVID/masks/COVID-2928.png,True,G02078,train,65677c5865e4f0974f71b8b2bf6c2a1e,d829c7161e5a7b38
COVID-2929.png,COVID,COVID/images/COVID-2929.png,COVID/masks/COVID-2929.png,True,G02079,train,bc6895bc8b1dd51e54551ae844b2048c,94c467933a69c57a
COVID-293.png,COVID,COVID/images/COVID-293.png,COVID/masks/COVID-293.png,True,G02080,train,2496bac87ad726b59616700e3a76b5df,897dc618495d3a4f
COVID-2930.png,COVID,COVID/images/COVID-2930.png,COVID/masks/COVID-2930.png,True,G02081,train,645054ed666151687ce125b83fda1dbf,897d871a0d5d386d
COVID-2931.png,COVID,COVID/images/COVID-2931.png,COVID/masks/COVID-2931.png,True,G02082,train,f8ebf1ef83fc1be2ac9c61f798cda6c4,8d2c474b1b5c4c5f
COVID-2932.png,COVID,COVID/images/COVID-2932.png,COVID/masks/COVID-2932.png,True,G02083,train,a3a318cee9d6338e352d20376983db1b,83680584d61f55ff
COVID-2933.png,COVID,COVID/images/COVID-2933.png,COVID/masks/COVID-2933.png,True,G02084,train,ddafac09b0d658fafdca8260313d044e,886d065e0e7f1a79
COVID-2934.png,COVID,COVID/images/COVID-2934.png,COVID/masks/COVID-2934.png,True,G02085,train,2ec586d3e1565bbd64db19612d4f7471,81a87e847a0f6d7a
COVID-2935.png,COVID,COVID/images/COVID-2935.png,COVID/masks/COVID-2935.png,True,G02086,val,14d949618b7eb1e85a6a68a0fd4c54bc,81695e965a192d5f
COVID-2936.png,COVID,COVID/images/COVID-2936.png,COVID/masks/COVID-2936.png,True,G02087,val,94ed4f5d820c19e99d6f2c6e6496de33,892fd60a965b1d5a
COVID-2937.png,COVID,COVID/images/COVID-2937.png,COVID/masks/COVID-2937.png,True,G02088,test,ad605b1e55c998d97c136891d1df9899,dc2ec35b5f421e44
COVID-2938.png,COVID,COVID/images/COVID-2938.png,COVID/masks/COVID-2938.png,True,G02089,train,62e3f06124780b667f6d774d53daa72d,896e9610f60add5e
COVID-2939.png,COVID,COVID/images/COVID-2939.png,COVID/masks/COVID-2939.png,True,G02090,test,e7508db75cba30d2b2c621f0bc23d2d1,817f461a19acbeb1
COVID-294.png,COVID,COVID/images/COVID-294.png,COVID/masks/COVID-294.png,True,G02091,test,7192a9906e7df27e6d2f324aade03c3b,d969c54e7c42794a
COVID-2940.png,COVID,COVID/images/COVID-2940.png,COVID/masks/COVID-2940.png,True,G02092,train,1c15cf96d66c93c540518aeb3bf9bb98,957c47d21b4a865e
COVID-2941.png,COVID,COVID/images/COVID-2941.png,COVID/masks/COVID-2941.png,True,G02093,train,40d4e00a3448fc5d9d095afc990c7253,d46e031b0b4d3e7c
COVID-2942.png,COVID,COVID/images/COVID-2942.png,COVID/masks/COVID-2942.png,True,G02093,train,40d4e00a3448fc5d9d095afc990c7253,d46e031b0b4d3e7c
COVID-2943.png,COVID,COVID/images/COVID-2943.png,COVID/masks/COVID-2943.png,True,G02094,train,d1511c4c2ca8fa644c4896de857af817,8179e79e12a61eb4
COVID-2944.png,COVID,COVID/images/COVID-2944.png,COVID/masks/COVID-2944.png,True,G02095,train,ad9d0fc304e80428d9577fc4033a4186,d5ae07194e283f6c
COVID-2945.png,COVID,COVID/images/COVID-2945.png,COVID/masks/COVID-2945.png,True,G02096,train,d4537e736d1b44bea6eed7750e3ee82e,897bc71e1c4f1e18
COVID-2946.png,COVID,COVID/images/COVID-2946.png,COVID/masks/COVID-2946.png,True,G02096,train,d6a93853a10d9d1b6574abe6cb9295e3,897bc71e1c4f1e18
COVID-2947.png,COVID,COVID/images/COVID-2947.png,COVID/masks/COVID-2947.png,True,G02097,train,a14cf32dffb7d3f3e2bb808e4af39968,9978165a796929ab
COVID-2948.png,COVID,COVID/images/COVID-2948.png,COVID/masks/COVID-2948.png,True,G02098,train,6fb68cb073ab1c783289b83672750f21,817d06161b4f7979
COVID-2949.png,COVID,COVID/images/COVID-2949.png,COVID/masks/COVID-2949.png,True,G02099,train,09e8e1afec2dd6727ce0082fffb52403,d52e475b1d4c425e
COVID-295.png,COVID,COVID/images/COVID-295.png,COVID/masks/COVID-295.png,True,G00362,train,44fefb055e28a8aae18981313eac9ef1,cc2dc252db531a5b
COVID-2950.png,COVID,COVID/images/COVID-2950.png,COVID/masks/COVID-2950.png,True,G02100,train,14f63e19f323d12ca6f06f406a527488,dd2ed2431b5d481d
COVID-2951.png,COVID,COVID/images/COVID-2951.png,COVID/masks/COVID-2951.png,True,G02101,val,f21151db7e220e531300049c609cf220,d93b461a194d1b3d
COVID-2952.png,COVID,COVID/images/COVID-2952.png,COVID/masks/COVID-2952.png,True,G02102,train,8a01f92b569c1b012b35b20b3b70ce3f,d92b061e1c4e3b7c
COVID-2953.png,COVID,COVID/images/COVID-2953.png,COVID/masks/COVID-2953.png,True,G02103,val,9937c5f24a716c8bc0046c674f7bf421,d42ed34b4d4c4f1c
COVID-2954.png,COVID,COVID/images/COVID-2954.png,COVID/masks/COVID-2954.png,True,G02104,train,afbb0f5216ce365f507c5fd9eeb0703f,d06ec21b4e4d1b4f
COVID-2955.png,COVID,COVID/images/COVID-2955.png,COVID/masks/COVID-2955.png,True,G02105,test,1c9252a99ac5388495244e63fefd0561,9834471b132f3a6f
COVID-2956.png,COVID,COVID/images/COVID-2956.png,COVID/masks/COVID-2956.png,True,G02106,train,27896d5d9a9a8bdd37a8cbe509ae1722,c8354b1b0e2db8ed
COVID-2957.png,COVID,COVID/images/COVID-2957.png,COVID/masks/COVID-2957.png,True,G02107,train,7f62b6947dcdc4dbb8df804ac6648758,88f58752835ef239
COVID-2958.png,COVID,COVID/images/COVID-2958.png,COVID/masks/COVID-2958.png,True,G02108,test,2822c05b4306026c5ec9f8ea3d0735e2,dc38d34d786d7848
COVID-2959.png,COVID,COVID/images/COVID-2959.png,COVID/masks/COVID-2959.png,True,G02109,train,61719998727e1fa49bb33fd2400f6078,dd3ac30f7c197c08
COVID-296.png,COVID,COVID/images/COVID-296.png,COVID/masks/COVID-296.png,True,G02110,train,56064163afee2e906f24026fe10120f9,983bc7961ece185a
COVID-2960.png,COVID,COVID/images/COVID-2960.png,COVID/masks/COVID-2960.png,True,G02111,val,9a7b5533d2c9f18d4dd02b720fd0a1d9,d979870a180d3bbe
COVID-2961.png,COVID,COVID/images/COVID-2961.png,COVID/masks/COVID-2961.png,True,G02112,test,f77e8fc671e416469458114e431b3733,81f84605124f7f5f
COVID-2962.png,COVID,COVID/images/COVID-2962.png,COVID/masks/COVID-2962.png,True,G02113,train,9ef79421bdb289d1ecb6c356e16151a9,c4bd4347994f461b
COVID-2963.png,COVID,COVID/images/COVID-2963.png,COVID/masks/COVID-2963.png,True,G02114,train,19aa62fded1a2c22d8fd552598b3cc66,936f8518ccb75887
COVID-2964.png,COVID,COVID/images/COVID-2964.png,COVID/masks/COVID-2964.png,True,G02115,train,d48d9554e48b3275eb2e3a5f1bb20a79,854c4e7b616be0cd
COVID-2965.png,COVID,COVID/images/COVID-2965.png,COVID/masks/COVID-2965.png,True,G02116,train,5a66054d810c3df20ab1fd82c7cfa10a,917b461ef8692939
COVID-2966.png,COVID,COVID/images/COVID-2966.png,COVID/masks/COVID-2966.png,True,G02117,val,18acfbd43a8b768684e4a4e21bee6512,d42e43cb1dc44f5c
COVID-2967.png,COVID,COVID/images/COVID-2967.png,COVID/masks/COVID-2967.png,True,G02118,train,1cae470168c5337c7b6154297b4a9edd,d978961a793939a1
COVID-2968.png,COVID,COVID/images/COVID-2968.png,COVID/masks/COVID-2968.png,True,G02119,train,d6958b73b6ec7e0f4ab3f2241fe11184,d933d614984f3539
COVID-2969.png,COVID,COVID/images/COVID-2969.png,COVID/masks/COVID-2969.png,True,G02120,train,6d3abe0f6f5267a979432a3dac48c60d,d9bbd6ca28c22bc4
COVID-297.png,COVID,COVID/images/COVID-297.png,COVID/masks/COVID-297.png,True,G02121,test,9a873eab17e9cddd43babc48f61d0db9,cc6c121b4f1f5955
COVID-2970.png,COVID,COVID/images/COVID-2970.png,COVID/masks/COVID-2970.png,True,G02122,val,d4238d58693ff371dab9cd47f29ba09f,cc2c834bace4cd97
COVID-2971.png,COVID,COVID/images/COVID-2971.png,COVID/masks/COVID-2971.png,True,G02123,train,49ef2ea487fee9938bcd67f2472d6393,d56e43594e691c4e
COVID-2972.png,COVID,COVID/images/COVID-2972.png,COVID/masks/COVID-2972.png,True,G02124,val,4d01f081d91579f065b3e21052c8946b,c4bca3af91c34f90
COVID-2973.png,COVID,COVID/images/COVID-2973.png,COVID/masks/COVID-2973.png,True,G02125,test,411990f3e4d0d3199eebc298a64445dd,d95b433839a9b4b4
COVID-2974.png,COVID,COVID/images/COVID-2974.png,COVID/masks/COVID-2974.png,True,G02126,test,7586a225541165140d69eea672057d42,dc2cc21bd9475a39
COVID-2975.png,COVID,COVID/images/COVID-2975.png,COVID/masks/COVID-2975.png,True,G02127,train,395fa78f3882b26566715b1765653a47,9d2c071a1d4f5c5d
COVID-2976.png,COVID,COVID/images/COVID-2976.png,COVID/masks/COVID-2976.png,True,G02128,train,00ea6f3878347cad4b729538421e20d2,817a8786163edf1c
COVID-2977.png,COVID,COVID/images/COVID-2977.png,COVID/masks/COVID-2977.png,True,G02129,val,6c1606bc460c24e5159ce6533618c0f2,807d87565a4b9f0b
COVID-2978.png,COVID,COVID/images/COVID-2978.png,COVID/masks/COVID-2978.png,True,G02130,train,94dec752dc7d4a93b7a215539158059c,c07d47561a4b4f1e
COVID-2979.png,COVID,COVID/images/COVID-2979.png,COVID/masks/COVID-2979.png,True,G02131,train,a4fdd47b3f7aa9dc7f35da7fe640e440,d42623cb2dd02bdd
COVID-298.png,COVID,COVID/images/COVID-298.png,COVID/masks/COVID-298.png,True,G02132,train,f820d95e21f034ba2ee6810952827b99,dd2cc2431b4d4b1f
COVID-2980.png,COVID,COVID/images/COVID-2980.png,COVID/masks/COVID-2980.png,True,G02133,train,e3acc9db8e117639d93c5f89e763418b,dc2ec34dc30b5d0e
COVID-2981.png,COVID,COVID/images/COVID-2981.png,COVID/masks/COVID-2981.png,True,G02134,train,34c7f0aeeda6add62e373413ec10b787,fc2cc3c31b4d4d1c
COVID-2982.png,COVID,COVID/images/COVID-2982.png,COVID/masks/COVID-2982.png,True,G02135,train,19c308733a2bd19e524f492ecd4588eb,d4287b63574b4d4c
COVID-2983.png,COVID,COVID/images/COVID-2983.png,COVID/masks/COVID-2983.png,True,G02136,train,42eaf8599b30f1b4d7216402c8d7677f,d42e23cd2ed42ed4
COVID-2984.png,COVID,COVID/images/COVID-2984.png,COVID/masks/COVID-2984.png,True,G02137,train,8d36bd4d07b73697d7bd68fb4af1ae2e,dc2ec35b1b403e1e
COVID-2985.png,COVID,COVID/images/COVID-2985.png,COVID/masks/COVID-2985.png,True,G02138,train,023641d355b981eaac08f70f92350649,807dd3131ba53e3c
COVID-2986.png,COVID,COVID/images/COVID-2986.png,COVID/masks/COVID-2986.png,True,G02139,train,cd25fadfac71beec3fcc949d68e5d6a1,cd2cde4b0d3f21b0
COVID-2987.png,COVID,COVID/images/COVID-2987.png,COVID/masks/COVID-2987.png,True,G02140,train,15eaa01327d80ceb74ed9c607ec2c775,dd2a521f5d481db4
COVID-2988.png,COVID,COVID/images/COVID-2988.png,COVID/masks/COVID-2988.png,True,G02141,train,12f241ce133b5f7beb10eee16b13feab,d52d46c20ec62bdd
COVID-2989.png,COVID,COVID/images/COVID-2989.png,COVID/masks/COVID-2989.png,True,G02142,train,9dd99ee2ed4a4e23c968e6ba3a94530e,917b869692c74756
COVID-299.png,COVID,COVID/images/COVID-299.png,COVID/masks/COVID-299.png,True,G02143,train,63e1e2eca69a42e45428e35a4eba95c6,937ac4131b5b7272
COVID-2990.png,COVID,COVID/images/COVID-2990.png,COVID/masks/COVID-2990.png,True,G02144,train,af874c48f42424c3031e211926c5dfe9,d93bc61e1a465b19
COVID-2991.png,COVID,COVID/images/COVID-2991.png,COVID/masks/COVID-2991.png,True,G02145,train,d0b9cef56dfe1c6769f828f95269c451,d13d07561a196f2b
COVID-2992.png,COVID,COVID/images/COVID-2992.png,COVID/masks/COVID-2992.png,True,G02146,val,9d3ce12207802a3d4ea2a5ca0a701f3c,d52b430a2a6a3ef6
COVID-2993.png,COVID,COVID/images/COVID-2993.png,COVID/masks/COVID-2993.png,True,G02147,test,052a4a06d137e3a979cd3bd243457760,dc3da2838d87831f
COVID-2994.png,COVID,COVID/images/COVID-2994.png,COVID/masks/COVID-2994.png,True,G02148,test,a4e559149ceb913e1b6875b5570f8be3,995b161a18bb38e7
COVID-2995.png,COVID,COVID/images/COVID-2995.png,COVID/masks/COVID-2995.png,True,G02149,val,52245c09396290ff159990b56d1e0aa0,dc4f0b4d64eac2d2
COVID-2996.png,COVID,COVID/images/COVID-2996.png,COVID/masks/COVID-2996.png,True,G02150,test,c369a9f2e4588b060d47ebf527290a44,d07bc64a5849793d
COVID-2997.png,COVID,COVID/images/COVID-2997.png,COVID/masks/COVID-2997.png,True,G02151,train,093cf56ec91d6b929cd1d462216487a9,946c075f1f4c7c68
COVID-2998.png,COVID,COVID/images/COVID-2998.png,COVID/masks/COVID-2998.png,True,G02152,val,98ba5f1dcc319675646947f92080aa4e,c1394e46586dec3b
COVID-2999.png,COVID,COVID/images/COVID-2999.png,COVID/masks/COVID-2999.png,True,G02153,val,6d702d04322c34f49220181c543f569d,9845471e4b5f1e4b
COVID-3.png,COVID,COVID/images/COVID-3.png,COVID/masks/COVID-3.png,True,G02154,train,a8ec5328e56b07819e3c4f4453a74ba1,892a860e8e9f9bd5
COVID-30.png,COVID,COVID/images/COVID-30.png,COVID/masks/COVID-30.png,True,G02155,train,6ce095cdfd871d2af846ac5eaed7044c,dd7c460b0f4c383d
COVID-300.png,COVID,COVID/images/COVID-300.png,COVID/masks/COVID-300.png,True,G02156,train,c36a3f08ce915ebb73f0e2f0f925598e,ed6dd21a0d4a1acd
COVID-3000.png,COVID,COVID/images/COVID-3000.png,COVID/masks/COVID-3000.png,True,G02157,train,7340f2a2b873d9b369504a264e25d959,956e831b4d595d4c
COVID-3001.png,COVID,COVID/images/COVID-3001.png,COVID/masks/COVID-3001.png,True,G02158,train,1a979a4d20a6441a965fb36faee260d9,dc2ac30f0c4a1feb
COVID-3002.png,COVID,COVID/images/COVID-3002.png,COVID/masks/COVID-3002.png,True,G02159,train,66aa5cbc8fbf3e6758f66c4078f2e9ae,dc1c4b0b3c3d999c
COVID-3003.png,COVID,COVID/images/COVID-3003.png,COVID/masks/COVID-3003.png,True,G02160,train,af20febb4c2fa107fc2c0c2bbdfe7a33,f90d8618c9b9ace5
COVID-3004.png,COVID,COVID/images/COVID-3004.png,COVID/masks/COVID-3004.png,True,G02161,train,b8e1fcf1dbda3c5b173f2f2b98d2c53f,9179c614164e5b5f
COVID-3005.png,COVID,COVID/images/COVID-3005.png,COVID/masks/COVID-3005.png,True,G02162,train,8984cf098fe1fb838cd1eec7860753ac,91f88611175f3b1e
COVID-3006.png,COVID,COVID/images/COVID-3006.png,COVID/masks/COVID-3006.png,True,G02163,train,5e1d36d0761a8d1423acf4f0fa450dfd,d130cb031b4f39db
COVID-3007.png,COVID,COVID/images/COVID-3007.png,COVID/masks/COVID-3007.png,True,G02164,train,532acc16a3fb90f85af57922a66f61f5,956886965bcd3978
COVID-3008.png,COVID,COVID/images/COVID-3008.png,COVID/masks/COVID-3008.png,True,G02165,train,1368a8bca23e124638eb2d4d577069d8,dc1ed34b4e4a6b28
COVID-3009.png,COVID,COVID/images/COVID-3009.png,COVID/masks/COVID-3009.png,True,G02166,train,0d06274a931d3e29499d1ff084766d31,d52b47071b591d1d
COVID-301.png,COVID,COVID/images/COVID-301.png,COVID/masks/COVID-301.png,True,G02167,val,b0390e6be6bc881a33fd712d051bc30b,dc28c31f6c3c3d4a
COVID-3010.png,COVID,COVID/images/COVID-3010.png,COVID/masks/COVID-3010.png,True,G02168,test,d426b0517f23a5437a905fab467a3d9e,dd2dd2031f611a1f
COVID-3011.png,COVID,COVID/images/COVID-3011.png,COVID/masks/COVID-3011.png,True,G02169,test,5f2798d7be362357f809536c22aaa465,dcaf8b0d460d4c4f
COVID-3012.png,COVID,COVID/images/COVID-3012.png,COVID/masks/COVID-3012.png,True,G02170,test,637b8a8ed84117b90428b59657da7682,dc0cc30d2cecbd3c
COVID-3013.png,COVID,COVID/images/COVID-3013.png,COVID/masks/COVID-3013.png,True,G02171,train,600da708fd34d698504d09382312b1ff,887b87961ae95b38
COVID-3014.png,COVID,COVID/images/COVID-3014.png,COVID/masks/COVID-3014.png,True,G02172,train,4f7a8602611b38fb58273b3e993255fb,887d971b1e4e5a19
COVID-3015.png,COVID,COVID/images/COVID-3015.png,COVID/masks/COVID-3015.png,True,G02173,train,bce95436a02f16a226a6742ff74c2bb8,c46dd3425e5a4a2f
COVID-3016.png,COVID,COVID/images/COVID-3016.png,COVID/masks/COVID-3016.png,True,G02174,train,11f7ed84095819fdbf73857bf05e6bd8,d42d434b4f4f0d0f
COVID-3017.png,COVID,COVID/images/COVID-3017.png,COVID/masks/COVID-3017.png,True,G02175,train,7d4ca00c37651d54b967a43a035409b7,c13f961b1c461b5d
COVID-3018.png,COVID,COVID/images/COVID-3018.png,COVID/masks/COVID-3018.png,True,G02176,train,5338174c2576e0d4b0e1c2dc5105f8c8,9859c71eda431a4f
COVID-3019.png,COVID,COVID/images/COVID-3019.png,COVID/masks/COVID-3019.png,True,G02177,train,0415cd6ad63d19b84a50ae14f5f6c945,d43e51d30f434e4d
COVID-302.png,COVID,COVID/images/COVID-302.png,COVID/masks/COVID-302.png,True,G02178,train,4f73540da04f1b18a0d5e8ab7849779c,dd0c461a4dcdd8f8
COVID-3020.png,COVID,COVID/images/COVID-3020.png,COVID/masks/COVID-3020.png,True,G02179,train,e94110d705ab242c7514ca6121e85bae,903c2f877acd6b81
COVID-3021.png,COVID,COVID/images/COVID-3021.png,COVID/masks/COVID-3021.png,True,G02180,train,9131accf6ee7226073afd462638f65df,ea3f2f4e91d0a145
COVID-3022.png,COVID,COVID/images/COVID-3022.png,COVID/masks/COVID-3022.png,True,G02181,train,d46eaff756567b2bfff5b18401ff85fc,d42c4b0b1f2c7c3d
COVID-3023.png,COVID,COVID/images/COVID-3023.png,COVID/masks/COVID-3023.png,True,G02182,train,21a8704617c45415f4f97f8b0e2f3d31,d428535b4b4d6b4b
COVID-3024.png,COVID,COVID/images/COVID-3024.png,COVID/masks/COVID-3024.png,True,G02183,train,1307918df043666c249a258bf3f8925d,d63d4949094f2d3d
COVID-3025.png,COVID,COVID/images/COVID-3025.png,COVID/masks/COVID-3025.png,True,G02184,train,1824545735d528c5bf4bbb3f8bd0ba73,817b121a3bb8b9f2
COVID-3026.png,COVID,COVID/images/COVID-3026.png,COVID/masks/COVID-3026.png,True,G02185,val,26b9bdb43798517099c242b1e5c1b349,994e0f79e9f48087
COVID-3027.png,COVID,COVID/images/COVID-3027.png,COVID/masks/COVID-3027.png,True,G02186,val,5aa2e29387260c9da18d9bc6c0227810,817f061e1a5e5e78
COVID-3028.png,COVID,COVID/images/COVID-3028.png,COVID/masks/COVID-3028.png,True,G02187,train,6f238efaf78b44d3bb49a34fa09678f3,d829d25c79793ab0
COVID-3029.png,COVID,COVID/images/COVID-3029.png,COVID/masks/COVID-3029.png,True,G02188,train,a5507f7426f03d44e06ef38c0bccc998,9979c718851f195e
COVID-303.png,COVID,COVID/images/COVID-303.png,COVID/masks/COVID-303.png,True,G02189,val,bcda21b3feeae521fd66cdb60f9f9ddc,857c1753195c6c7c
COVID-3030.png,COVID,COVID/images/COVID-3030.png,COVID/masks/COVID-3030.png,True,G02190,test,2b423499189a410af6937909947d9d00,d037462e3e692d2e
COVID-3031.png,COVID,COVID/images/COVID-3031.png,COVID/masks/COVID-3031.png,True,G02191,test,1c6de62e7bff90b46fdc4c9912fdfcf0,94e466cb2f96380f
COVID-3032.png,COVID,COVID/images/COVID-3032.png,COVID/masks/COVID-3032.png,True,G02192,test,ac5d5c49b4d74e5bca62d21270bf42fd,946e47130f5f1c4e
COVID-3033.png,COVID,COVID/images/COVID-3033.png,COVID/masks/COVID-3033.png,True,G02193,train,a3708f5d697d96533dfca08116173624,817b07161e5d5f19
COVID-3034.png,COVID,COVID/images/COVID-3034.png,COVID/masks/COVID-3034.png,True,G02194,train,160b65941d2abd5de5dc653d13fa9fde,917fc61a184d5b59
COVID-3035.png,COVID,COVID/images/COVID-3035.png,COVID/masks/COVID-3035.png,True,G02195,train,163448d55dab6e257718c960841570f8,993dc3961e4d1e1c
COVID-3036.png,COVID,COVID/images/COVID-3036.png,COVID/masks/COVID-3036.png,True,G02196,train,9f7d3a64347967155ec4c277da7b248a,d43cc35b5a471b1a
COVID-3037.png,COVID,COVID/images/COVID-3037.png,COVID/masks/COVID-3037.png,True,G02197,train,33db32a7922bffdd34de19ddc4ada20a,817847161a4d5f5f
COVID-3038.png,COVID,COVID/images/COVID-3038.png,COVID/masks/COVID-3038.png,True,G02198,train,0b915a51b2445e50f43dc3587cbd4ce7,d47d82190d5e3b1e
COVID-3039.png,COVID,COVID/images/COVID-3039.png,COVID/masks/COVID-3039.png,True,G02199,train,4418c3d4c863fcc785031188bbf1af19,81781e0a1a5f6d7b
COVID-304.png,COVID,COVID/images/COVID-304.png,COVID/masks/COVID-304.png,True,G02200,train,0d23ea6b9ba08e20d117d864d02ba4a5,dd69421643c38fcb
COVID-3040.png,COVID,COVID/images/COVID-3040.png,COVID/masks/COVID-3040.png,True,G02201,train,5ba5bffee55d9b89391c277bd0765f9e,817c6e865a996ec5
COVID-3041.png,COVID,COVID/images/COVID-3041.png,COVID/masks/COVID-3041.png,True,G02202,train,caaef051fa8421ac7201744c6b599851,81795f86161d585f
COVID-3042.png,COVID,COVID/images/COVID-3042.png,COVID/masks/COVID-3042.png,True,G02203,train,a6739a38ff77ac1af8c077729c475c2f,d475d30f643c6c0e
COVID-3043.png,COVID,COVID/images/COVID-3043.png,COVID/masks/COVID-3043.png,True,G02204,train,ccd3aac3e787310c4c212941188c6662,993b47165e491a1f
COVID-3044.png,COVID,COVID/images/COVID-3044.png,COVID/masks/COVID-3044.png,True,G02205,val,00ed8725167b04f87bf58207bcd917aa,d979c71657471243
COVID-3045.png,COVID,COVID/images/COVID-3045.png,COVID/masks/COVID-3045.png,True,G02206,val,2283813d1072a465f940eeba2004faaa,887e061a5b6d595d
COVID-3046.png,COVID,COVID/images/COVID-3046.png,COVID/masks/COVID-3046.png,True,G02207,train,ab1b17b098e68a00d7bfbe24b39b2db2,917d86563b850f3a
COVID-3047.png,COVID,COVID/images/COVID-3047.png,COVID/masks/COVID-3047.png,True,G02208,train,55d614eb1411541bda16da825210bbc8,9168071e1e9f1e97
COVID-3048.png,COVID,COVID/images/COVID-3048.png,COVID/masks/COVID-3048.png,True,G02209,val,a3a6dfaf34bc326d9fa76c5926d8b9e2,9129c7c71f47861b
COVID-3049.png,COVID,COVID/images/COVID-3049.png,COVID/masks/COVID-3049.png,True,G02210,train,03091cf4c471bdc2ec80802f71260be6,9dbec31b461e184e
COVID-305.png,COVID,COVID/images/COVID-305.png,COVID/masks/COVID-305.png,True,G02211,test,4d6f1618a2148595bdadc3bde3f49f33,d62f51492e2f2726
COVID-3050.png,COVID,COVID/images/COVID-3050.png,COVID/masks/COVID-3050.png,True,G02212,train,71e43738168d64172a2d51926ee45f36,dd4cc34b4b4e0a5d
COVID-3051.png,COVID,COVID/images/COVID-3051.png,COVID/masks/COVID-3051.png,True,G02213,train,ae02eecbbaa6915c04c080cd4394f34d,885dc6571b4d0f1b
COVID-3052.png,COVID,COVID/images/COVID-3052.png,COVID/masks/COVID-3052.png,True,G02214,train,c1fb64896f0f164c44509866b6a91af0,dd3d40c61b594c9e
COVID-3053.png,COVID,COVID/images/COVID-3053.png,COVID/masks/COVID-3053.png,True,G02215,val,c17e918c8e1fe1656e6ad4553554e086,d93d41961b494f1d
COVID-3054.png,COVID,COVID/images/COVID-3054.png,COVID/masks/COVID-3054.png,True,G02216,test,cb79847cb86f3b22921e0b3abbbe839d,823714177b60e3eb
COVID-3055.png,COVID,COVID/images/COVID-3055.png,COVID/masks/COVID-3055.png,True,G02217,train,99896af214a90cdb755a48f5c4f9d02f,cc0c42535f6c6c7e
COVID-3056.png,COVID,COVID/images/COVID-3056.png,COVID/masks/COVID-3056.png,True,G02218,test,c4e74d473a35b6ee52253533819eeb6d,d42e554b4a4b6e2e
COVID-3057.png,COVID,COVID/images/COVID-3057.png,COVID/masks/COVID-3057.png,True,G02219,val,b08f549f9eda3232fd88d553dfd2a8b6,992ac61f594d1d1d
COVID-3058.png,COVID,COVID/images/COVID-3058.png,COVID/masks/COVID-3058.png,True,G02220,train,9fd28ce4344d33f6b0101b5c83d930a5,d57c061e596dc878
COVID-3059.png,COVID,COVID/images/COVID-3059.png,COVID/masks/COVID-3059.png,True,G02221,train,c06d2f8c16f13fe176df6b10d4dbc92c,917a0c585b78edc5
COVID-306.png,COVID,COVID/images/COVID-306.png,COVID/masks/COVID-306.png,True,G02222,train,76be90a3b3542df139642d47b9535a60,c92df4920fe55c1a
COVID-3060.png,COVID,COVID/images/COVID-3060.png,COVID/masks/COVID-3060.png,True,G02223,test,2db0a5d8e3d60ae21ae285381ea8bd32,d97f961e18525c54
COVID-3061.png,COVID,COVID/images/COVID-3061.png,COVID/masks/COVID-3061.png,True,G02224,train,1e0c6784605f46abbe5d80bdcba19831,d52e435f092f0a3d
COVID-3062.png,COVID,COVID/images/COVID-3062.png,COVID/masks/COVID-3062.png,True,G02225,train,1ec159ac6e41cf17a5dafd38ff95eaf9,817b8714120f7b3f
COVID-3063.png,COVID,COVID/images/COVID-3063.png,COVID/masks/COVID-3063.png,True,G02226,train,25d51f8f241636af77e070cb4e8c5fe8,94d1932c0bea1bbb
COVID-3064.png,COVID,COVID/images/COVID-3064.png,COVID/masks/COVID-3064.png,True,G02227,train,b7328886d1089e44bcc1c8378214010d,9d28c61fc30f3c4b
COVID-3065.png,COVID,COVID/images/COVID-3065.png,COVID/masks/COVID-3065.png,True,G02228,val,ce8266676951916a08b2517e5e5041c9,993b871e18495f5a
COVID-3066.png,COVID,COVID/images/COVID-3066.png,COVID/masks/COVID-3066.png,True,G02229,test,f14085bd5b4c0109b2de2d5e51a2ecdf,d43cc2531f451e1f
COVID-3067.png,COVID,COVID/images/COVID-3067.png,COVID/masks/COVID-3067.png,True,G02230,val,b32cc01a4ca4f57e7c6ca736bba7d121,81785fc61e1a5a5b
COVID-3068.png,COVID,COVID/images/COVID-3068.png,COVID/masks/COVID-3068.png,True,G02231,test,51889981bcfac61f145aab0cf1ffea05,d42fc3170d41796d
COVID-3069.png,COVID,COVID/images/COVID-3069.png,COVID/masks/COVID-3069.png,True,G02232,train,38f320b3ab4cf1afbe901bad7e30e9d0,815b041e1b7b5a7b
COVID-307.png,COVID,COVID/images/COVID-307.png,COVID/masks/COVID-307.png,True,G02233,train,2fd1d767129825734f2a665f9eba129a,986b87169e193e5a
COVID-3070.png,COVID,COVID/images/COVID-3070.png,COVID/masks/COVID-3070.png,True,G02234,train,5014199f153dfe41be224b53e1dac5c2,dd68d30b4b4a1e69
COVID-3071.png,COVID,COVID/images/COVID-3071.png,COVID/masks/COVID-3071.png,True,G02235,train,a1f8d6c7d174830cf6b6581b1b6c4371,8c6ad31d65b6e0e1
COVID-3072.png,COVID,COVID/images/COVID-3072.png,COVID/masks/COVID-3072.png,True,G02236,train,8cb3b55e0d127dbe7c8e37e10d4323ee,d43e071b1a4c5e79
COVID-3073.png,COVID,COVID/images/COVID-3073.png,COVID/masks/COVID-3073.png,True,G02237,train,6ca465355c2d36ea5bf4c46b058488eb,990e161e155e5e5e
COVID-3074.png,COVID,COVID/images/COVID-3074.png,COVID/masks/COVID-3074.png,True,G02238,train,977795f1213bbfbdecb23e004a5f025f,dd2c531f421b702f
COVID-3075.png,COVID,COVID/images/COVID-3075.png,COVID/masks/COVID-3075.png,True,G02239,train,f5d274ccdf24f877068d4555f154b582,997b8516961f1a8d
COVID-3076.png,COVID,COVID/images/COVID-3076.png,COVID/masks/COVID-3076.png,True,G02240,train,da3e0c4946b6377f6ae3e9114538f0fa,80137fc45e78e963
COVID-3077.png,COVID,COVID/images/COVID-3077.png,COVID/masks/COVID-3077.png,True,G02241,train,80320a65a6b8134c8ee30789acfefd77,91695e16130e7a3f
COVID-3078.png,COVID,COVID/images/COVID-3078.png,COVID/masks/COVID-3078.png,True,G02242,train,c0821b679b417db13aab2d01592fa637,d46ec31b194d1e5c
COVID-3079.png,COVID,COVID/images/COVID-3079.png,COVID/masks/COVID-3079.png,True,G02243,train,cf519415000d904e3a4b7a1115925f3b,816cc69b1a4f7b1a
COVID-308.png,COVID,COVID/images/COVID-308.png,COVID/masks/COVID-308.png,True,G02244,train,b4fa0689e874ab33df1f8cc15ae34588,81797e967ac06ae1
COVID-3080.png,COVID,COVID/images/COVID-3080.png,COVID/masks/COVID-3080.png,True,G02245,train,485bb5f08df63649cbc6a5bde5b954c1,d43d42c71f4b071b
COVID-3081.png,COVID,COVID/images/COVID-3081.png,COVID/masks/COVID-3081.png,True,G02246,val,8c1ad69a496a0593012064ca81585760,992f86171f071a5d
COVID-3082.png,COVID,COVID/images/COVID-3082.png,COVID/masks/COVID-3082.png,True,G02247,val,a2f677ff33e104522170b35d7d39873a,d129c6461e1d4f5b
COVID-3083.png,COVID,COVID/images/COVID-3083.png,COVID/masks/COVID-3083.png,True,G02248,test,39d48629750ac11ceed5da581b0967cb,817d86165b051f7b
COVID-3084.png,COVID,COVID/images/COVID-3084.png,COVID/masks/COVID-3084.png,True,G02249,test,6d0620ba98411774500bd0cf631a94dd,807fc782923acb6d
COVID-3085.png,COVID,COVID/images/COVID-3085.png,COVID/masks/COVID-3085.png,True,G02250,test,5ee9aa21cc80018280efc024b220a79d,cd2e535f42491e4b
COVID-3086.png,COVID,COVID/images/COVID-3086.png,COVID/masks/COVID-3086.png,True,G02251,train,584915dc30a62016300f73eff1af21f1,856a4e17115f7e19
COVID-3087.png,COVID,COVID/images/COVID-3087.png,COVID/masks/COVID-3087.png,True,G02252,train,8043d66073ccc2b33e56733681247b95,c07f5f53084a6f29
COVID-3088.png,COVID,COVID/images/COVID-3088.png,COVID/masks/COVID-3088.png,True,G02253,train,e3dfb43703c681addedc8ebde080acd6,817987961a899f1f
COVID-3089.png,COVID,COVID/images/COVID-3089.png,COVID/masks/COVID-3089.png,True,G02254,train,ac1cae6b1bbb0855544ccb7ead7db9aa,9968165b1f29b1e6
COVID-309.png,COVID,COVID/images/COVID-309.png,COVID/masks/COVID-309.png,True,G02255,train,e30698a7f2a9467f65fcbf79562c0e89,c875d20fd55a4758
COVID-3090.png,COVID,COVID/images/COVID-3090.png,COVID/masks/COVID-3090.png,True,G02256,train,636ad16394648d4d30b6330b6200d51b,917f86061f0d1f1d
COVID-3091.png,COVID,COVID/images/COVID-3091.png,COVID/masks/COVID-3091.png,True,G02257,train,8b3861d1a8ebc8cb95da344d12654fa0,9877471e703a233e
COVID-3092.png,COVID,COVID/images/COVID-3092.png,COVID/masks/COVID-3092.png,True,G02258,train,c525d1cf89818d962e66e9772fee0541,817b161a585d6b79
COVID-3093.png,COVID,COVID/images/COVID-3093.png,COVID/masks/COVID-3093.png,True,G02259,train,7946ad182e11111d2a6bce1d3e981373,883f0616195f5b5d
COVID-3094.png,COVID,COVID/images/COVID-3094.png,COVID/masks/COVID-3094.png,True,G02260,train,e7fd5ff4ec48d2f746b6afab35518ace,952e475f5c406e0f
COVID-3095.png,COVID,COVID/images/COVID-3095.png,COVID/masks/COVID-3095.png,True,G02261,val,f4747ef08ccbd81b0b05b3340991e2ee,817b17961819cf5b
COVID-3096.png,COVID,COVID/images/COVID-3096.png,COVID/masks/COVID-3096.png,True,G02262,train,9757c4eeefa1d5da9d299bd6bcb5b3d7,837a15041e5a5f5f
COVID-3097.png,COVID,COVID/images/COVID-3097.png,COVID/masks/COVID-3097.png,True,G02263,train,b586fa87ff9ebad91ad2bd6fcb8e169d,997e16382d27a4e5
COVID-3098.png,COVID,COVID/images/COVID-3098.png,COVID/masks/COVID-3098.png,True,G02264,train,a93e3e486112317306e99e40165e358b,916806175f4e5a7b
COVID-3099.png,COVID,COVID/images/COVID-3099.png,COVID/masks/COVID-3099.png,True,G02265,train,6732caaac655d7ffad3040b8e8e440f5,8d2d06165e5c7e69
COVID-31.png,COVID,COVID/images/COVID-31.png,COVID/masks/COVID-31.png,True,G02266,train,6b9aaa56834e2bdfa880e27fbf1d8f33,dd60934fd01ac71e
COVID-310.png,COVID,COVID/images/COVID-310.png,COVID/masks/COVID-310.png,True,G02267,val,ddede401ca016b502a06778b02d8f624,8c2cc353491f4f4f
COVID-3100.png,COVID,COVID/images/COVID-3100.png,COVID/masks/COVID-3100.png,True,G02268,val,7d4c780562f4b952afe3662cb458dee5,886f1618491d4f7f
COVID-3101.png,COVID,COVID/images/COVID-3101.png,COVID/masks/COVID-3101.png,True,G02269,test,9f6b614dc7909343f7733f09321debf6,9d68c2175b1f1c59
COVID-3102.png,COVID,COVID/images/COVID-3102.png,COVID/masks/COVID-3102.png,True,G02270,test,1d11e2381c97275cb706dd1782acae82,897f861a1d4f1a4d
COVID-3103.png,COVID,COVID/images/COVID-3103.png,COVID/masks/COVID-3103.png,True,G02271,test,54f74acd17497439d64f050fe70b71d4,893f461a194f394f
COVID-3104.png,COVID,COVID/images/COVID-3104.png,COVID/masks/COVID-3104.png,True,G02272,train,7f5e1283a934e091d6e5ff9f9256b18b,817a5618195f7b39
COVID-3105.png,COVID,COVID/images/COVID-3105.png,COVID/masks/COVID-3105.png,True,G02273,train,6d49b3537391771aeb660483d728ddc8,8169851a175e7b1f
COVID-3106.png,COVID,COVID/images/COVID-3106.png,COVID/masks/COVID-3106.png,True,G02274,train,ab12bc7fdd75e052ce0230766e3f77d9,d17943435b586a6b
COVID-3107.png,COVID,COVID/images/COVID-3107.png,COVID/masks/COVID-3107.png,True,G02275,train,2f780d313ee97fa7b994433e24530963,dd2e425b1d4c4b3c
COVID-3108.png,COVID,COVID/images/COVID-3108.png,COVID/masks/COVID-3108.png,True,G02276,val,d8d45cfaf1f9f77cc7891f8a2d7860fe,81782e965a793d3a
COVID-3109.png,COVID,COVID/images/COVID-3109.png,COVID/masks/COVID-3109.png,True,G02277,train,9119a5e499466b5ade01ae74163dabff,c6274b4d2e276566
COVID-311.png,COVID,COVID/images/COVID-311.png,COVID/masks/COVID-311.png,True,G02278,train,9fdd77128cd0abba3b692f20a7408f5f,d109565e7a796961
COVID-3110.png,COVID,COVID/images/COVID-3110.png,COVID/masks/COVID-3110.png,True,G02279,train,a5b90c7bde277323943a458966719cec,cd6ec65b1747180d
COVID-3111.png,COVID,COVID/images/COVID-3111.png,COVID/masks/COVID-3111.png,True,G02280,train,a7d8ebf364a377479d03db77b9e8417b,896e865a495f4a4f
COVID-3112.png,COVID,COVID/images/COVID-3112.png,COVID/masks/COVID-3112.png,True,G02281,train,7f8726e5d13bccc6a4e8eb0226e7fa88,d43d420f1b8a6fb8
COVID-3113.png,COVID,COVID/images/COVID-3113.png,COVID/masks/COVID-3113.png,True,G02282,train,16795bb1b3f093cde58f7b6fb3bcdfeb,d46c07121bcf1b3b
COVID-3114.png,COVID,COVID/images/COVID-3114.png,COVID/masks/COVID-3114.png,True,G02283,train,3035c2bf6d66b3a1553d38f487aafea2,d42f461f186c6e78
COVID-3115.png,COVID,COVID/images/COVID-3115.png,COVID/masks/COVID-3115.png,True,G02284,train,9a789d51c3f9e619c68c2670f5c0ac1d,d52bc61f184f1e58
COVID-3116.png,COVID,COVID/images/COVID-3116.png,COVID/masks/COVID-3116.png,True,G02285,train,9d76aff96e1ada48c980ec4331780b9c,c2d03fc32d3dc83c
COVID-3117.png,COVID,COVID/images/COVID-3117.png,COVID/masks/COVID-3117.png,True,G02286,test,948f5719c0f9de942e93eff086894e59,817bc714145b1f3b
COVID-3118.png,COVID,COVID/images/COVID-3118.png,COVID/masks/COVID-3118.png,True,G02287,val,cf98de46db897e484324aafbd3d4f480,90bd875099471b5f
COVID-3119.png,COVID,COVID/images/COVID-3119.png,COVID/masks/COVID-3119.png,True,G02288,train,05610e88c29e24a0e9816799fd098a56,812926c63fc43fda
COVID-312.png,COVID,COVID/images/COVID-312.png,COVID/masks/COVID-312.png,True,G02289,test,86ff0c25cde98021e37e1df47d0b1fa6,d93bc35f1e58184a
COVID-3120.png,COVID,COVID/images/COVID-3120.png,COVID/masks/COVID-3120.png,True,G02290,val,2f22616c2bbe160f8c4bdbee7c809415,817846561b4e7b5b
COVID-3121.png,COVID,COVID/images/COVID-3121.png,COVID/masks/COVID-3121.png,True,G02291,test,03412b4a8e1a7bb5d18603a6e1dd829f,957a06061b5d0f7d
COVID-3122.png,COVID,COVID/images/COVID-3122.png,COVID/masks/COVID-3122.png,True,G02292,train,9dd758edde82c57c5220598f13fb706f,817c06571b5d0f7c
COVID-3123.png,COVID,COVID/images/COVID-3123.png,COVID/masks/COVID-3123.png,True,G02293,train,9775e0d7a3e333e1c8fef88300b24bac,d16f82a71a017f55
COVID-3124.png,COVID,COVID/images/COVID-3124.png,COVID/masks/COVID-3124.png,True,G02294,test,01877a5cfa225ae4a1273ad9bccd8384,903df2130e82d75f
COVID-3125.png,COVID,COVID/images/COVID-3125.png,COVID/masks/COVID-3125.png,True,G02295,train,e93f297188934152214a3c90967045d5,817cce5e1c4d385b
COVID-3126.png,COVID,COVID/images/COVID-3126.png,COVID/masks/COVID-3126.png,True,G02296,train,2c23909042ac469d688da29f4f175c2d,917a5719697bb086
COVID-3127.png,COVID,COVID/images/COVID-3127.png,COVID/masks/COVID-3127.png,True,G02297,train,77b37332980e9ad4cc52bb8d00afe29e,c879561a5c1e5d0f
COVID-3128.png,COVID,COVID/images/COVID-3128.png,COVID/masks/COVID-3128.png,True,G02298,train,29ab43ff793099612efd823d21fc7ace,9c22537d64bbe194
COVID-3129.png,COVID,COVID/images/COVID-3129.png,COVID/masks/COVID-3129.png,True,G02299,train,af1e48a30eadd3a1ec71a37fbd32932d,d42a632b2c333b79
COVID-313.png,COVID,COVID/images/COVID-313.png,COVID/masks/COVID-313.png,True,G02300,train,366914112af725432c599216e24fa1b8,9c2c8213474f8f5f
COVID-3130.png,COVID,COVID/images/COVID-3130.png,COVID/masks/COVID-3130.png,True,G02301,train,a92c1f6b9c1c47a95d36d950200c13f3,917bc61e18595f19
COVID-3131.png,COVID,COVID/images/COVID-3131.png,COVID/masks/COVID-3131.png,True,G02302,train,320f6dd77940b84ace7328fc46cf0cf3,d43c53c79a454717
COVID-3132.png,COVID,COVID/images/COVID-3132.png,COVID/masks/COVID-3132.png,True,G02303,train,85f9f9ac5ab271f1e9c51d68acd96e0b,8178071c797d396d
COVID-3133.png,COVID,COVID/images/COVID-3133.png,COVID/masks/COVID-3133.png,True,G02304,train,4e26f263316cb4643e71c3a5ef552d0f,8079565a591b3b3d
COVID-3134.png,COVID,COVID/images/COVID-3134.png,COVID/masks/COVID-3134.png,True,G02305,train,7940b3f0878ea1d7ad08c07a9f05bc19,d42f82434f464f4f
COVID-3135.png,COVID,COVID/images/COVID-3135.png,COVID/masks/COVID-3135.png,True,G02306,val,68b27d7da4452651a5b9c6b4407b283f,d12f561e1d4d1b19
COVID-3136.png,COVID,COVID/images/COVID-3136.png,COVID/masks/COVID-3136.png,True,G02307,train,dcafb812c0bb0b8de7e4379ea99cdd3a,d46e03d70b4d8b5c
COVID-3137.png,COVID,COVID/images/COVID-3137.png,COVID/masks/COVID-3137.png,True,G02308,val,a8838bbb1ace15e45c9d17db9f2b823b,953e061b59ec687c
COVID-3138.png,COVID,COVID/images/COVID-3138.png,COVID/masks/COVID-3138.png,True,G02309,val,3121b55374e88e344ce9349fb6b89167,916906161f5ede5a
COVID-3139.png,COVID,COVID/images/COVID-3139.png,COVID/masks/COVID-3139.png,True,G02310,test,0e6e37a31498350efec0289a28bdd219,93491e9c7858717b
COVID-314.png,COVID,COVID/images/COVID-314.png,COVID/masks/COVID-314.png,True,G02311,test,047006c840f320a138f48f4b5671206f,d42e125f4c2e2e6d
COVID-3140.png,COVID,COVID/images/COVID-3140.png,COVID/masks/COVID-3140.png,True,G02312,train,8e7368933d400defbe24f3481ab80b44,987d871e045f384f
COVID-3141.png,COVID,COVID/images/COVID-3141.png,COVID/masks/COVID-3141.png,True,G02312,train,8e7368933d400defbe24f3481ab80b44,987d871e045f384f
COVID-3142.png,COVID,COVID/images/COVID-3142.png,COVID/masks/COVID-3142.png,True,G02312,train,8e7368933d400defbe24f3481ab80b44,987d871e045f384f
COVID-3143.png,COVID,COVID/images/COVID-3143.png,COVID/masks/COVID-3143.png,True,G02312,train,8e7368933d400defbe24f3481ab80b44,987d871e045f384f
COVID-3144.png,COVID,COVID/images/COVID-3144.png,COVID/masks/COVID-3144.png,True,G02312,train,8e7368933d400defbe24f3481ab80b44,987d871e045f384f
COVID-3145.png,COVID,COVID/images/COVID-3145.png,COVID/masks/COVID-3145.png,True,G02313,train,5f6e74e46059743b084261aa68594f4c,8c3393491f4d1c7d
COVID-3146.png,COVID,COVID/images/COVID-3146.png,COVID/masks/COVID-3146.png,True,G02314,train,11b381a778ec4320c32cf92209ad6521,987de7521821397d
COVID-3147.png,COVID,COVID/images/COVID-3147.png,COVID/masks/COVID-3147.png,True,G02315,train,d71a26955732d01624b1e28a628804c8,d42e03494d7c7ce6
COVID-3148.png,COVID,COVID/images/COVID-3148.png,COVID/masks/COVID-3148.png,True,G02316,train,b478204cb31e954c8d4a9603e1ffa38a,d63449496d696e66
COVID-3149.png,COVID,COVID/images/COVID-3149.png,COVID/masks/COVID-3149.png,True,G02317,train,94b515faa65cf55ef9aadab0ff4c3595,816906871f5d4f1f
COVID-315.png,COVID,COVID/images/COVID-315.png,COVID/masks/COVID-315.png,True,G00469,train,f5e6e6b493f4ec4282b4e0d02e4c8626,ed3d1796188907ba
COVID-3150.png,COVID,COVID/images/COVID-3150.png,COVID/masks/COVID-3150.png,True,G02318,train,6b9177d4ac4de239bffb5554eb69ce4c,917b1a1b7979c2c2
COVID-3151.png,COVID,COVID/images/COVID-3151.png,COVID/masks/COVID-3151.png,True,G02319,train,ebb089b6477afceb12b435ccd4db67df,d03d42571b5a6f0b
COVID-3152.png,COVID,COVID/images/COVID-3152.png,COVID/masks/COVID-3152.png,True,G02320,train,8157ac98e431ae73139ea5e1128c4730,997907161a5d4f5a
COVID-3153.png,COVID,COVID/images/COVID-3153.png,COVID/masks/COVID-3153.png,True,G02321,train,ed00d0685867e18de52456fe8280f76c,d42dc2d71bd44f18
COVID-3154.png,COVID,COVID/images/COVID-3154.png,COVID/masks/COVID-3154.png,True,G02322,train,01f2b4021880e18ec8842a47655a9d75,d92b061e196939fc
COVID-3155.png,COVID,COVID/images/COVID-3155.png,COVID/masks/COVID-3155.png,True,G02323,train,595e1f8487d3c985b4e6785144474639,c979061e1a59795b
COVID-3156.png,COVID,COVID/images/COVID-3156.png,COVID/masks/COVID-3156.png,True,G02324,train,f4240bce4de9539f3c148054515e9ffc,cd2ec24b0de40eed
COVID-3157.png,COVID,COVID/images/COVID-3157.png,COVID/masks/COVID-3157.png,True,G02325,train,9469eae6e4fbb9c86b5f533321528632,d42d5f535f084f0a
COVID-3158.png,COVID,COVID/images/COVID-3158.png,COVID/masks/COVID-3158.png,True,G02326,val,75f164246d74405506c6ff93b44f149f,9968171a957a71e9
COVID-3159.png,COVID,COVID/images/COVID-3159.png,COVID/masks/COVID-3159.png,True,G02327,val,80c777fee548dff3254ca7dba7b0729c,9969855e9c5f185c
COVID-316.png,COVID,COVID/images/COVID-316.png,COVID/masks/COVID-316.png,True,G00414,train,177eb9750442d0f97874fbc48aabff0c,d86d561e486d3c3c
COVID-3160.png,COVID,COVID/images/COVID-3160.png,COVID/masks/COVID-3160.png,True,G02328,val,e8941d5eb45e57a5eb225895bd0a74cc,c17c96425d5e4b4b
COVID-3161.png,COVID,COVID/images/COVID-3161.png,COVID/masks/COVID-3161.png,True,G02329,test,b88e320d52b5490345750dba180d88f3,dd2ec24b0bc64b1d
COVID-3162.png,COVID,COVID/images/COVID-3162.png,COVID/masks/COVID-3162.png,True,G02330,test,5646d528cdfa1b3aa3b7ec62a03ad84c,816d5e1d584f293b
COVID-3163.png,COVID,COVID/images/COVID-3163.png,COVID/masks/COVID-3163.png,True,G02331,test,e6e964526a544224b211b736dcc0cf9e,ce2a512eaea68fc1
COVID-3164.png,COVID,COVID/images/COVID-3164.png,COVID/masks/COVID-3164.png,True,G02332,train,421f000de09e84121e61997faee59084,893c461b4d5e1b6d
COVID-3165.png,COVID,COVID/images/COVID-3165.png,COVID/masks/COVID-3165.png,True,G02333,train,d4fbd71047c7dd0b717c1701eab67a9c,d43d4b475d4e2e28
COVID-3166.png,COVID,COVID/images/COVID-3166.png,COVID/masks/COVID-3166.png,True,G02334,train,da0d3eb45fcd35f95a7b5e574616e25f,dd58d21b5d0a1b2d
COVID-3167.png,COVID,COVID/images/COVID-3167.png,COVID/masks/COVID-3167.png,True,G02335,train,bfa349dd314abfcdeafda66563fd2400,917d86125a5f1a5e
COVID-3168.png,COVID,COVID/images/COVID-3168.png,COVID/masks/COVID-3168.png,True,G02336,val,8c9503acd4557c43cbcf217a81b22f44,9179061e1f5f5e18
COVID-3169.png,COVID,COVID/images/COVID-3169.png,COVID/masks/COVID-3169.png,True,G02337,test,418b57cc6dfed8b0132cabca54576568,cc42c35b4b585f47
COVID-317.png,COVID,COVID/images/COVID-317.png,COVID/masks/COVID-317.png,True,G02338,train,ee037baa29ad7d8bfe9bd8c3deca0bee,d42e06d30bdb4b8b
COVID-3170.png,COVID,COVID/images/COVID-3170.png,COVID/masks/COVID-3170.png,True,G02339,train,971ea424bbccb2204e1996541bcd7622,d86dc7161ec3711c
COVID-3171.png,COVID,COVID/images/COVID-3171.png,COVID/masks/COVID-3171.png,True,G02340,train,975f3b033ab28aeb1f8b18b64e6a2b5f,cc3c4f4c58ecb8e4
COVID-3172.png,COVID,COVID/images/COVID-3172.png,COVID/masks/COVID-3172.png,True,G02341,train,ee0cd1261aeacf83d1f1f66a47e10bf6,d42e025b195e4f79
COVID-3173.png,COVID,COVID/images/COVID-3173.png,COVID/masks/COVID-3173.png,True,G02342,train,59d3ade2ade794a72344dbc4f693b477,d02d47c25f1d4e2d
COVID-3174.png,COVID,COVID/images/COVID-3174.png,COVID/masks/COVID-3174.png,True,G02343,train,0e292be26c7b1331c9adfe81f6366fc7,d839550b1a3abab3
COVID-3175.png,COVID,COVID/images/COVID-3175.png,COVID/masks/COVID-3175.png,True,G02344,train,cffca4f9dccc7226f19f06ee54ca4017,d17d57d31a4c0d38
COVID-3176.png,COVID,COVID/images/COVID-3176.png,COVID/masks/COVID-3176.png,True,G02345,train,ea11ca2f9dee590efb23b3878edb8270,817f04545a1f1f5b
COVID-3177.png,COVID,COVID/images/COVID-3177.png,COVID/masks/COVID-3177.png,True,G02346,train,eaef6b186bb8370070ac9f6498631cf8,d52a620b2e2a2f3f
COVID-3178.png,COVID,COVID/images/COVID-3178.png,COVID/masks/COVID-3178.png,True,G02347,train,cfbc299aaba318e06a97f755001d42b0,dd2e464b59454f38
COVID-3179.png,COVID,COVID/images/COVID-3179.png,COVID/masks/COVID-3179.png,True,G02348,val,26551e94140b1462abb819338fadb3d2,917806865f1d4b5f
COVID-318.png,COVID,COVID/images/COVID-318.png,COVID/masks/COVID-318.png,True,G02349,val,b519765276e3ff16099c3b416a321a7a,c83f916d96521f94
COVID-3180.png,COVID,COVID/images/COVID-3180.png,COVID/masks/COVID-3180.png,True,G02350,test,58810275efacafbb8dcc6f8efd50c8a9,91784f861e199e4f
COVID-3181.png,COVID,COVID/images/COVID-3181.png,COVID/masks/COVID-3181.png,True,G02351,test,e6a2bc6ff664de1881b14fc6a48c452a,d92f861a5c4d594d
COVID-3182.png,COVID,COVID/images/COVID-3182.png,COVID/masks/COVID-3182.png,True,G02352,train,0c7b60e199170e4fe2944ee4e4255210,857c06561f5e2c6d
COVID-3183.png,COVID,COVID/images/COVID-3183.png,COVID/masks/COVID-3183.png,True,G02353,train,c44e00c7d77aa202327f847e20f0f231,817b17061e155b5f
COVID-3184.png,COVID,COVID/images/COVID-3184.png,COVID/masks/COVID-3184.png,True,G02354,train,fb607ffe3e3e1f9be36217e5da89f9ee,895b061e5a5e5a7a
COVID-3185.png,COVID,COVID/images/COVID-3185.png,COVID/masks/COVID-3185.png,True,G02355,val,9965ef1216604952a7548ceae1775b3d,d52c42c35f1a4f0f
COVID-3186.png,COVID,COVID/images/COVID-3186.png,COVID/masks/COVID-3186.png,True,G02356,train,9dcda580ae6b4034bed95dec1e7e362e,c03dc2d7594a475b
COVID-3187.png,COVID,COVID/images/COVID-3187.png,COVID/masks/COVID-3187.png,True,G02357,train,20c6b65d4d02725e7be434b93790f1c4,91695a16185fe57a
COVID-3188.png,COVID,COVID/images/COVID-3188.png,COVID/masks/COVID-3188.png,True,G02358,train,58c4a0c06483c78adaa6ba137337dbf2,dc2d4a431f4e6c6c
COVID-3189.png,COVID,COVID/images/COVID-3189.png,COVID/masks/COVID-3189.png,True,G02359,train,89c68999d07820a2f8ac28deb8bdd005,d839421e3e692d3d
COVID-319.png,COVID,COVID/images/COVID-319.png,COVID/masks/COVID-319.png,True,G02360,train,d9674072f8d9180b149d77dca9ab21b0,d40e430e0c6c6eff
COVID-3190.png,COVID,COVID/images/COVID-3190.png,COVID/masks/COVID-3190.png,True,G02361,train,de379b4e870c632e08e0c3822f5bf11b,d43e87470b4f1f18
COVID-3191.png,COVID,COVID/images/COVID-3191.png,COVID/masks/COVID-3191.png,True,G02362,test,8701c72e9e6159c8068d22f7e6d23168,d17a523a39a3151f
COVID-3192.png,COVID,COVID/images/COVID-3192.png,COVID/masks/COVID-3192.png,True,G02363,train,4981b87223cf98eb49acb73d424219cd,d969d5521a5d4b19
COVID-3193.png,COVID,COVID/images/COVID-3193.png,COVID/masks/COVID-3193.png,True,G02364,train,336b2260b0eadadc16af4c23267a4301,d529f5424a1f1e5a
COVID-3194.png,COVID,COVID/images/COVID-3194.png,COVID/masks/COVID-3194.png,True,G02365,test,b271fabdf90010ef4c476b2732695cfa,dd2c52831b4f4e1e
COVID-3195.png,COVID,COVID/images/COVID-3195.png,COVID/masks/COVID-3195.png,True,G02366,train,40d0f6eee7232be0ba583ab19ae49971,982d87161a4f1d5f
COVID-3196.png,COVID,COVID/images/COVID-3196.png,COVID/masks/COVID-3196.png,True,G02367,train,cf14dce9b75a85f442d126ca8c543a3d,956c0e58fde9e8c0
COVID-3197.png,COVID,COVID/images/COVID-3197.png,COVID/masks/COVID-3197.png,True,G02368,train,2962c9d011443559bcda5f8fd7a11f21,99580c1e1b4f276f
COVID-3198.png,COVID,COVID/images/COVID-3198.png,COVID/masks/COVID-3198.png,True,G02369,val,1ee496658c07ca51bef7381d1ae4f5d2,dd2e870f0d4c495d
COVID-3199.png,COVID,COVID/images/COVID-3199.png,COVID/masks/COVID-3199.png,True,G02370,val,1b5c859e64f804a97d8852a55e946a94,817fc61a194f196d
COVID-32.png,COVID,COVID/images/COVID-32.png,COVID/masks/COVID-32.png,True,G01439,val,38762702a3b185bab98ce09df30b5d95,dd2cc21bc44f1d47
COVID-320.png,COVID,COVID/images/COVID-320.png,COVID/masks/COVID-320.png,True,G02371,train,2cde7aeb53610802060bcd79d9f61f0e,fc2dc01287c61fc7
COVID-3200.png,COVID,COVID/images/COVID-3200.png,COVID/masks/COVID-3200.png,True,G02372,test,dd1eeb60291e2fded13fefb3c7e7f9ca,b96d871a851e185f
COVID-3201.png,COVID,COVID/images/COVID-3201.png,COVID/masks/COVID-3201.png,True,G02373,train,67231402d202b80c06471743eae1f3bb,c2c13d9f39393239
COVID-3202.png,COVID,COVID/images/COVID-3202.png,COVID/masks/COVID-3202.png,True,G02374,train,528bd178da09daa969795d9a57d57acb,997dc61b510f1b49
COVID-3203.png,COVID,COVID/images/COVID-3203.png,COVID/masks/COVID-3203.png,True,G02375,train,a7263a8c749888565a4033f35802b4ef,817de6c61a594f46
COVID-3204.png,COVID,COVID/images/COVID-3204.png,COVID/masks/COVID-3204.png,True,G02376,train,4b78961d2c6a3b761a3081f4c30bb664,99181e5b79f086a7
COVID-3205.png,COVID,COVID/images/COVID-3205.png,COVID/masks/COVID-3205.png,True,G02377,train,b5323ce82caba5ee82cf0af91426d351,817b57161a5a1f1b
COVID-3206.png,COVID,COVID/images/COVID-3206.png,COVID/masks/COVID-3206.png,True,G02378,train,b685ecbec81355043f2e9bea55844adb,897d061a5d5b194f
COVID-3207.png,COVID,COVID/images/COVID-3207.png,COVID/masks/COVID-3207.png,True,G02379,train,91fc14a1600056eeac4a16fe41f4e010,9d2cc71b5c4f5843
COVID-3208.png,COVID,COVID/images/COVID-3208.png,COVID/masks/COVID-3208.png,True,G02379,train,91fc14a1600056eeac4a16fe41f4e010,9d2cc71b5c4f5843
COVID-3209.png,COVID,COVID/images/COVID-3209.png,COVID/masks/COVID-3209.png,True,G02380,train,3962f9961ed6e11d63a40c32446380d7,817b0c185b7b59cb
COVID-321.png,COVID,COVID/images/COVID-321.png,COVID/masks/COVID-321.png,True,G02360,train,d9674072f8d9180b149d77dca9ab21b0,d40e430e0c6c6eff
COVID-3210.png,COVID,COVID/images/COVID-3210.png,COVID/masks/COVID-3210.png,True,G02381,train,3912c8f32c3e39cd7596cdafda56a443,817a06da1f557a59
COVID-3211.png,COVID,COVID/images/COVID-3211.png,COVID/masks/COVID-3211.png,True,G02382,train,c6e8c0b08993a076255cb0bc975f997f,8168071e1fde5e59
COVID-3212.png,COVID,COVID/images/COVID-3212.png,COVID/masks/COVID-3212.png,True,G02382,train,c6e8c0b08993a076255cb0bc975f997f,8168071e1fde5e59
COVID-3213.png,COVID,COVID/images/COVID-3213.png,COVID/masks/COVID-3213.png,True,G02383,train,1ab669f5dd28ea0f92ec90609f20eb86,817f07861a1b8f4f
COVID-3214.png,COVID,COVID/images/COVID-3214.png,COVID/masks/COVID-3214.png,True,G02384,train,514eb8eac2e691199962d0fc93447851,d42a821e0f3a3ee7
COVID-3215.png,COVID,COVID/images/COVID-3215.png,COVID/masks/COVID-3215.png,True,G02385,train,f151c50b4ed17b85575bffc6e5821423,d52d0643031f5f5b
COVID-3216.png,COVID,COVID/images/COVID-3216.png,COVID/masks/COVID-3216.png,True,G02386,train,d2fa1e255580574fa6d3a9d29c017bc8,c828071f5f4d5f38
COVID-3217.png,COVID,COVID/images/COVID-3217.png,COVID/masks/COVID-3217.png,True,G02387,val,3a21b646020438a8ce4fce8cbfa4ef9c,d42e831d1c7c6c7c
COVID-3218.png,COVID,COVID/images/COVID-3218.png,COVID/masks/COVID-3218.png,True,G02388,val,eab244cb3d3c8fa6d0cc1966a4222387,817b06061e1b5f5f
COVID-3219.png,COVID,COVID/images/COVID-3219.png,COVID/masks/COVID-3219.png,True,G02389,val,3956f53056d68b4a16e7a7522e03d4c5,d42f42435f4d6e0e
COVID-322.png,COVID,COVID/images/COVID-322.png,COVID/masks/COVID-322.png,True,G02390,test,ae89f1c412aecae6927e411711147e43,b82d8616c31f1ec7
COVID-3220.png,COVID,COVID/images/COVID-3220.png,COVID/masks/COVID-3220.png,True,G02391,test,1ce60aa4a7624be5f928b8201a9f4574,d93f061a19493d5b
COVID-3221.png,COVID,COVID/images/COVID-3221.png,COVID/masks/COVID-3221.png,True,G02392,test,9104257b3b7527caf39817bb1109c0ee,c47e27520b4fac2d
COVID-3222.png,COVID,COVID/images/COVID-3222.png,COVID/masks/COVID-3222.png,True,G02393,train,97f0927dc25c65b007df3650e50b919d,987f871e18417bb8
COVID-3223.png,COVID,COVID/images/COVID-3223.png,COVID/masks/COVID-3223.png,True,G02393,train,97f0927dc25c65b007df3650e50b919d,987f871e18417bb8
COVID-3224.png,COVID,COVID/images/COVID-3224.png,COVID/masks/COVID-3224.png,True,G02394,train,d6561819bc4c72ccb4e3a1e7160c4bac,c56f5242486f6b69
COVID-3225.png,COVID,COVID/images/COVID-3225.png,COVID/masks/COVID-3225.png,True,G02395,train,d3d533c6dc1fcc8db8057d144e142340,cd0ed34b4d2a4d2e
COVID-3226.png,COVID,COVID/images/COVID-3226.png,COVID/masks/COVID-3226.png,True,G02396,train,d11a575692e87bbb42fb9b65d19867e1,d42d03434b1f2eee
COVID-3227.png,COVID,COVID/images/COVID-3227.png,COVID/masks/COVID-3227.png,True,G02397,train,a5ff7df0bcbce8d66ffc31000135d6df,d13b0b471a3879e9
COVID-3228.png,COVID,COVID/images/COVID-3228.png,COVID/masks/COVID-3228.png,True,G02398,train,2f4da6bd82f65be7020dafd665c3fb76,d929c2471b964f17
COVID-3229.png,COVID,COVID/images/COVID-3229.png,COVID/masks/COVID-3229.png,True,G02399,train,fa1bcb1ae745ff4ac4d738e432b5d029,9d611c1e5b597853
COVID-323.png,COVID,COVID/images/COVID-323.png,COVID/masks/COVID-323.png,True,G00139,train,a2aaa91edd10d32cffb77f624581f55b,dd28d3c35f55420b
COVID-3230.png,COVID,COVID/images/COVID-3230.png,COVID/masks/COVID-3230.png,True,G02400,train,3edc7119264ce1d37f52b217bd79a2e2,893d871e17471f1c
COVID-3231.png,COVID,COVID/images/COVID-3231.png,COVID/masks/COVID-3231.png,True,G02401,train,4829243cb6e6527b8db870b1ca532bfb,81695e16105f5f5e
COVID-3232.png,COVID,COVID/images/COVID-3232.png,COVID/masks/COVID-3232.png,True,G02402,train,904a50e77ff157cb8122ef1f8e07af4c,d4b427c31ac22f9b
COVID-3233.png,COVID,COVID/images/COVID-3233.png,COVID/masks/COVID-3233.png,True,G02403,train,178ab0ce5231fe3a62ff40e9e33885cc,d43d0bc2174f471b
COVID-3234.png,COVID,COVID/images/COVID-3234.png,COVID/masks/COVID-3234.png,True,G02404,train,7c4829ce1323c4b4ee36a8952d0478b1,d52856c7431a5b4f
COVID-3235.png,COVID,COVID/images/COVID-3235.png,COVID/masks/COVID-3235.png,True,G02405,val,555b6a59688ae5185e7205f289fb4454,9128065e1d5e7b79
COVID-3236.png,COVID,COVID/images/COVID-3236.png,COVID/masks/COVID-3236.png,True,G02406,val,665505b0ccc380386bdfae934514e26e,dd48c31e692f3ce0
COVID-3237.png,COVID,COVID/images/COVID-3237.png,COVID/masks/COVID-3237.png,True,G02407,val,48dd860a26c0b7e1aae87d3ba348d150,d43d5342db0bc2c7
COVID-3238.png,COVID,COVID/images/COVID-3238.png,COVID/masks/COVID-3238.png,True,G02408,train,0424d9ef30c71782ad493819ef7ab207,d52e46531b464f1d
COVID-3239.png,COVID,COVID/images/COVID-3239.png,COVID/masks/COVID-3239.png,True,G02409,train,36f85801f8c71b5a38721af4dd13fbd8,802d079e7b857dc9
COVID-324.png,COVID,COVID/images/COVID-324.png,COVID/masks/COVID-324.png,True,G02410,test,3c214fd59610a4d30d52ca842b457781,817d0786165a5e3f
COVID-3240.png,COVID,COVID/images/COVID-3240.png,COVID/masks/COVID-3240.png,True,G02411,train,2707fb3c23433720d0037319a978a754,89485e5a29adb5e6
COVID-3241.png,COVID,COVID/images/COVID-3241.png,COVID/masks/COVID-3241.png,True,G02412,train,d9636fc73d3c8ee19197a111fa8a98bb,d46e42d35b4d4a1d
COVID-3242.png,COVID,COVID/images/COVID-3242.png,COVID/masks/COVID-3242.png,True,G02413,test,767127ec1534ac1aa1b59c5cf9b290a4,81785e961a1b697d
COVID-3243.png,COVID,COVID/images/COVID-3243.png,COVID/masks/COVID-3243.png,True,G02414,train,0a65bdc7661be8560306e62beb8deb66,887f875a1a8f1a5a
COVID-3244.png,COVID,COVID/images/COVID-3244.png,COVID/masks/COVID-3244.png,True,G02415,train,41fc37b05c741f42e0d28a758fcbaf9c,997d861e184f1e5a
COVID-3245.png,COVID,COVID/images/COVID-3245.png,COVID/masks/COVID-3245.png,True,G02416,val,a8a9b9c2c144e6c899fb7b68586502c3,d43d838b0ac63ddc
COVID-3246.png,COVID,COVID/images/COVID-3246.png,COVID/masks/COVID-3246.png,True,G02417,train,0e3c7125b4b3d7e4060817c217401390,cd3a430f5f0e1e0b
COVID-3247.png,COVID,COVID/images/COVID-3247.png,COVID/masks/COVID-3247.png,True,G02418,test,793a69b1244d10425842fde6c374a6d5,c16e464e7c41eb96
COVID-3248.png,COVID,COVID/images/COVID-3248.png,COVID/masks/COVID-3248.png,True,G02419,train,09bb882064a5db95cde4d7cd555d8259,d17c47471a4b4e1b
COVID-3249.png,COVID,COVID/images/COVID-3249.png,COVID/masks/COVID-3249.png,True,G02420,train,01a278bd56a197838df1bb685823deec,817b07061e4f5f1b
COVID-325.png,COVID,COVID/images/COVID-325.png,COVID/masks/COVID-325.png,True,G02421,train,c596461ab08a66aee526b1c1f6c6bdca,d97f963a15a0b9a0
COVID-3250.png,COVID,COVID/images/COVID-3250.png,COVID/masks/COVID-3250.png,True,G02422,train,de7d1e1faa1e5ecf0e9f234d23e1a2db,d97fc6961859051d
COVID-3251.png,COVID,COVID/images/COVID-3251.png,COVID/masks/COVID-3251.png,True,G02423,train,67e29b97607eb41fa673e8d301ff01e0,dd2ad10b4f0a5d4d
COVID-3252.png,COVID,COVID/images/COVID-3252.png,COVID/masks/COVID-3252.png,True,G02424,train,81ad893fd2149d638af76db092764738,d96cc6125b4d7c58
COVID-3253.png,COVID,COVID/images/COVID-3253.png,COVID/masks/COVID-3253.png,True,G02425,train,aeed05f8582feeb134e964985bab687d,80a866d27f857f91
COVID-3254.png,COVID,COVID/images/COVID-3254.png,COVID/masks/COVID-3254.png,True,G02426,train,abf2306196980696233a1abf4f29a3ea,b969871e891d1a4f
COVID-3255.png,COVID,COVID/images/COVID-3255.png,COVID/masks/COVID-3255.png,True,G02427,train,de351c975b49d10d117846ac396bb048,d53cd20b92c79b61
COVID-3256.png,COVID,COVID/images/COVID-3256.png,COVID/masks/COVID-3256.png,True,G02428,test,61c1b46ee2813b139d6bec3fec69d2d2,d43d42431b6c4f3d
COVID-3257.png,COVID,COVID/images/COVID-3257.png,COVID/masks/COVID-3257.png,True,G02429,val,8bd3b1f8cf8fbcdd7b9e7b56df0148c8,dc2d424b0d4c6f3d
COVID-3258.png,COVID,COVID/images/COVID-3258.png,COVID/masks/COVID-3258.png,True,G02430,val,0f1d1f1650c38f360b3cbbf156c0486a,cc2cd35fd04b3c52
COVID-3259.png,COVID,COVID/images/COVID-3259.png,COVID/masks/COVID-3259.png,True,G02431,test,4b38c3f8cb8bd0f27b4c0d79af254903,913946d61b4a4e5f
COVID-326.png,COVID,COVID/images/COVID-326.png,COVID/masks/COVID-326.png,True,G02432,test,b7970a747e74cfdb2ad43d517fc6916d,cd45d20b4a0f4f4f
COVID-3260.png,COVID,COVID/images/COVID-3260.png,COVID/masks/COVID-3260.png,True,G02433,train,5cd686aa304052f2fd8c88be58035aa9,dd2e425f0f4e0f18
COVID-3261.png,COVID,COVID/images/COVID-3261.png,COVID/masks/COVID-3261.png,True,G02434,train,52cdbb3ff6c39e6fbb704b45e4b9dd38,81581efa79e1a587
COVID-3262.png,COVID,COVID/images/COVID-3262.png,COVID/masks/COVID-3262.png,True,G02435,train,0430a36f899329e144effe459b422c92,81780e1e5b5b6979
COVID-3263.png,COVID,COVID/images/COVID-3263.png,COVID/masks/COVID-3263.png,True,G02436,train,a58a2534cc4acf58d8a7113840a28fb5,c9b9c2171fc40f17
COVID-3264.png,COVID,COVID/images/COVID-3264.png,COVID/masks/COVID-3264.png,True,G02437,train,d7a53831d136775102722930f727f64f,81a97e943a857b1b
COVID-3265.png,COVID,COVID/images/COVID-3265.png,COVID/masks/COVID-3265.png,True,G02438,train,600b04c48fa724a73f6a38fd35ad99c6,952c065f5e6c78e4
COVID-3266.png,COVID,COVID/images/COVID-3266.png,COVID/masks/COVID-3266.png,True,G02439,train,8cad43eb7506953b65711afc23a25498,91384e1b574f391d
COVID-3267.png,COVID,COVID/images/COVID-3267.png,COVID/masks/COVID-3267.png,True,G02440,train,7b11571b1d8ac8a6199f42fa3b4c5e5b,c43ccf5353c20f1b
COVID-3268.png,COVID,COVID/images/COVID-3268.png,COVID/masks/COVID-3268.png,True,G02441,train,c1ab3528890d9adb8a7aab7115d8cd0e,896e065b574e194f
COVID-3269.png,COVID,COVID/images/COVID-3269.png,COVID/masks/COVID-3269.png,True,G02442,train,7c6c874399ee5d7b830072fd3e32d6d9,9160161a5add7add
COVID-327.png,COVID,COVID/images/COVID-327.png,COVID/masks/COVID-327.png,True,G02443,train,00db10b7593262c08263b119016f22bf,9859d61a49393fa5
COVID-3270.png,COVID,COVID/images/COVID-3270.png,COVID/masks/COVID-3270.png,True,G02444,train,912610b1146df11e40a7ea44693a2999,817fc65a194d1b59
COVID-3271.png,COVID,COVID/images/COVID-3271.png,COVID/masks/COVID-3271.png,True,G02445,train,4408a811f5cc68bdf44ab33da86c7990,817b5486141f5e5f
COVID-3272.png,COVID,COVID/images/COVID-3272.png,COVID/masks/COVID-3272.png,True,G02446,val,8b96b9aa45ba2ba1c20fbc8c9ba7bb6b,817b16065c5f5b78
COVID-3273.png,COVID,COVID/images/COVID-3273.png,COVID/masks/COVID-3273.png,True,G02447,val,280d8d2916258666ec4c62d33715ba78,897b8606195b5f59
COVID-3274.png,COVID,COVID/images/COVID-3274.png,COVID/masks/COVID-3274.png,True,G02448,val,c46f9aa654c35c7f355a13979df61205,957a60175e185b5b
COVID-3275.png,COVID,COVID/images/COVID-3275.png,COVID/masks/COVID-3275.png,True,G02449,test,2ac2badf314f1fa72b73c5f9ac2a3817,d636c14c4a6b0fcd
COVID-3276.png,COVID,COVID/images/COVID-3276.png,COVID/masks/COVID-3276.png,True,G02450,train,92e3cde714d35c5bedfee882a17119e0,d43d4352cb4b1bca
COVID-3277.png,COVID,COVID/images/COVID-3277.png,COVID/masks/COVID-3277.png,True,G02451,train,3ef9ad643fbcb9d8615bc829f405801c,916bc71296cf1a87
COVID-3278.png,COVID,COVID/images/COVID-3278.png,COVID/masks/COVID-3278.png,True,G02452,train,a72989506991799cfdde2bd2586c355a,9968071e4b5f1a4f
COVID-3279.png,COVID,COVID/images/COVID-3279.png,COVID/masks/COVID-3279.png,True,G02453,train,2a0ea1133a11ca8c3f0930f356c7bc3a,8c4d461a1e4f7c7c
COVID-328.png,COVID,COVID/images/COVID-328.png,COVID/masks/COVID-328.png,True,G02454,test,7bdf8a0411d30757de0b00997c9477bc,d96bc61e2a4e0f0d
COVID-3280.png,COVID,COVID/images/COVID-3280.png,COVID/masks/COVID-3280.png,True,G02455,test,795ee30735f0c401d290a04869f6c5d2,887d07d75a4c4d1d
COVID-3281.png,COVID,COVID/images/COVID-3281.png,COVID/masks/COVID-3281.png,True,G02456,train,31b30872b7710272514ad2957d352933,cc34430ad34f3e79
COVID-3282.png,COVID,COVID/images/COVID-3282.png,COVID/masks/COVID-3282.png,True,G02457,train,283c18374593775fa862a3753d18bcb6,dc3cc70b1583c70f
COVID-3283.png,COVID,COVID/images/COVID-3283.png,COVID/masks/COVID-3283.png,True,G02458,train,1a2025781f64af3f3732c84ebe18e2e5,d46e05d70a550f1f
COVID-3284.png,COVID,COVID/images/COVID-3284.png,COVID/masks/COVID-3284.png,True,G02459,train,9614deb1bab8e6b9568f40bdde896979,d92f161f184b39e1
COVID-3285.png,COVID,COVID/images/COVID-3285.png,COVID/masks/COVID-3285.png,True,G02460,train,9ae61057a52300197d38612f29eba938,956c0f5d5c286d6a
COVID-3286.png,COVID,COVID/images/COVID-3286.png,COVID/masks/COVID-3286.png,True,G02461,train,369ae454daa8e699aea1e891318f51e5,9168465e1b5f6a39
COVID-3287.png,COVID,COVID/images/COVID-3287.png,COVID/masks/COVID-3287.png,True,G02462,train,11e43b740836b5afdbca2696c591159e,d10b521a2b3bbf29
COVID-3288.png,COVID,COVID/images/COVID-3288.png,COVID/masks/COVID-3288.png,True,G02463,train,0cd0bb06120a9c5a62668219fa5ac84f,817b8514185f5f5b
COVID-3289.png,COVID,COVID/images/COVID-3289.png,COVID/masks/COVID-3289.png,True,G02464,val,9c3cacf29f16c9d97f9a3a415c699872,943c03c71e1d4d5f
COVID-329.png,COVID,COVID/images/COVID-329.png,COVID/masks/COVID-329.png,True,G02465,val,3102dbce2369bd4b38f630d7a71db311,cd18530f0d0b2faf
COVID-3290.png,COVID,COVID/images/COVID-3290.png,COVID/masks/COVID-3290.png,True,G02466,train,24fbf1b8d1d24e55893fdfcad92b5bb2,917be7c616185555
COVID-3291.png,COVID,COVID/images/COVID-3291.png,COVID/masks/COVID-3291.png,True,G02467,test,dec37d2e72a2fd36ef47300b3d8a1b50,d422c30f471e6f47
COVID-3292.png,COVID,COVID/images/COVID-3292.png,COVID/masks/COVID-3292.png,True,G02468,val,34948bc851d4c1254b2448b25dd5b523,987b071e185b78cb
COVID-3293.png,COVID,COVID/images/COVID-3293.png,COVID/masks/COVID-3293.png,True,G02469,test,cedb1b3d4d8748c9dd1b1aee89c67e62,d4bdc3990ac63dc2
COVID-3294.png,COVID,COVID/images/COVID-3294.png,COVID/masks/COVID-3294.png,True,G02470,val,bea520baa1dae644733fb82a015c9b83,d4dc2bd52bd52b08
COVID-3295.png,COVID,COVID/images/COVID-3295.png,COVID/masks/COVID-3295.png,True,G02471,train,98905011afe5aafff29112b8f19d9169,917b57061e18597b
COVID-3296.png,COVID,COVID/images/COVID-3296.png,COVID/masks/COVID-3296.png,True,G02472,train,37b7bda3074d5c892095a6aaa391b96a,807f961e5b5b7848
COVID-3297.png,COVID,COVID/images/COVID-3297.png,COVID/masks/COVID-3297.png,True,G02473,train,c706999e603806ceedd9e5e3dad9afba,d42852d75a5f425d
COVID-3298.png,COVID,COVID/images/COVID-3298.png,COVID/masks/COVID-3298.png,True,G02474,train,13e38943c57770d1fe413482917b709c,817c06161f5d5e5e
COVID-3299.png,COVID,COVID/images/COVID-3299.png,COVID/masks/COVID-3299.png,True,G02474,train,13e38943c57770d1fe413482917b709c,817c06161f5d5e5e
COVID-33.png,COVID,COVID/images/COVID-33.png,COVID/masks/COVID-33.png,True,G02475,train,45ab3920207f014df280d839ab696524,91a98615d74e1ad7
COVID-330.png,COVID,COVID/images/COVID-330.png,COVID/masks/COVID-330.png,True,G02476,test,361d23d657398bf945d3010d1e96e5fc,987f84101a9bbce7
COVID-3300.png,COVID,COVID/images/COVID-3300.png,COVID/masks/COVID-3300.png,True,G02477,train,56aa23dafb03c3156066f9e344cb3b00,904f163b69747665
COVID-3301.png,COVID,COVID/images/COVID-3301.png,COVID/masks/COVID-3301.png,True,G02478,train,14af9d92ce389edd30622f4b103787ff,d42ec31b4f686e2c
COVID-3302.png,COVID,COVID/images/COVID-3302.png,COVID/masks/COVID-3302.png,True,G02479,train,a04ad0bdbdc92a60d018835b98a1642e,d42f434f0e4e2c2f
COVID-3303.png,COVID,COVID/images/COVID-3303.png,COVID/masks/COVID-3303.png,True,G02480,train,d934f01a639c6e104d769143769598c5,d42d43570b0e5f1b
COVID-3304.png,COVID,COVID/images/COVID-3304.png,COVID/masks/COVID-3304.png,True,G02481,train,4b8f437adde986e259f8483726028bf3,d8b5d18aa2bfa18c
COVID-3305.png,COVID,COVID/images/COVID-3305.png,COVID/masks/COVID-3305.png,True,G02482,train,c01bd5504baf1c2c348a3068ed2878c6,dd4c53130b4d2e1f
COVID-3306.png,COVID,COVID/images/COVID-3306.png,COVID/masks/COVID-3306.png,True,G02483,train,686f2eff43aff494d20fb15c996945cb,d42e025b5deca8bc
COVID-3307.png,COVID,COVID/images/COVID-3307.png,COVID/masks/COVID-3307.png,True,G02484,train,34be61079dd79ef0452e522e8d94b68a,d42e42571b5d4e5c
COVID-3308.png,COVID,COVID/images/COVID-3308.png,COVID/masks/COVID-3308.png,True,G02485,train,9ede361d6d8c0c996926419765739994,d53c02431d5f2f3c
COVID-3309.png,COVID,COVID/images/COVID-3309.png,COVID/masks/COVID-3309.png,True,G02486,train,d10fc0b58d68043a20ed814526c5f7a8,d0fc2fc20bca0e3e
COVID-331.png,COVID,COVID/images/COVID-331.png,COVID/masks/COVID-331.png,True,G02487,val,d554f86cfb261e27adc51fdbf50b574c,9869e5163ee530e6
COVID-3310.png,COVID,COVID/images/COVID-3310.png,COVID/masks/COVID-3310.png,True,G02488,val,3c8b3a41ad15e72a67224a642c5cc251,c62e534b696f2c2c
COVID-3311.png,COVID,COVID/images/COVID-3311.png,COVID/masks/COVID-3311.png,True,G02489,test,68397fb4aea5bc62f15f6a0379dfbdfe,805b521b192f3dbd
COVID-3312.png,COVID,COVID/images/COVID-3312.png,COVID/masks/COVID-3312.png,True,G02490,test,207af72522dca676f32c4dbf21e10cb5,817f161a59697a39
COVID-3313.png,COVID,COVID/images/COVID-3313.png,COVID/masks/COVID-3313.png,True,G02491,test,4ba1b88dd0544e278e89b3bf4bb5248a,d42f534b4e4f0a4e
COVID-3314.png,COVID,COVID/images/COVID-3314.png,COVID/masks/COVID-3314.png,True,G02492,train,10e27c8afae29437ee9358d9f2f60f9f,d87dc35e43091a5b
COVID-3315.png,COVID,COVID/images/COVID-3315.png,COVID/masks/COVID-3315.png,True,G02493,train,6babdbf1263c64d17c5354f3bb5fe6b4,9939c71f14161e5d
COVID-3316.png,COVID,COVID/images/COVID-3316.png,COVID/masks/COVID-3316.png,True,G02494,train,519d55a8da2675b5cddd83cadc997573,d4ad0f5d430f290f
COVID-3317.png,COVID,COVID/images/COVID-3317.png,COVID/masks/COVID-3317.png,True,G02495,train,2ea42e96963aa579cddc17a2217513a4,9178c65a154b3b6e
COVID-3318.png,COVID,COVID/images/COVID-3318.png,COVID/masks/COVID-3318.png,True,G02496,train,ff12b0f4ff2223b3f031a2f95f07f036,895e0e7968e1a5a7
COVID-3319.png,COVID,COVID/images/COVID-3319.png,COVID/masks/COVID-3319.png,True,G02497,train,50ceb3b5586a736d11b16ac4318d50a2,803cd55e5c545e5d
COVID-332.png,COVID,COVID/images/COVID-332.png,COVID/masks/COVID-332.png,True,G02498,train,7aa88e6484deecb6d27a6b5ee51b76db,dd3ad24f54585c4a
COVID-3320.png,COVID,COVID/images/COVID-3320.png,COVID/masks/COVID-3320.png,True,G02499,train,a2e2942cc45720ca99e00b9a53e2acf3,c17b53511b4d2b1b
COVID-3321.png,COVID,COVID/images/COVID-3321.png,COVID/masks/COVID-3321.png,True,G02500,train,66b4a326b353d4dce0cf538d9bccc028,d13b56581a6b3939
COVID-3322.png,COVID,COVID/images/COVID-3322.png,COVID/masks/COVID-3322.png,True,G02501,train,b7bb78bb828c50994e143c068aaa9845,cd2cd00b5b451f1f
COVID-3323.png,COVID,COVID/images/COVID-3323.png,COVID/masks/COVID-3323.png,True,G02502,train,769d325c842424e043b75d766ea7f0fc,d96e465f186d0a39
COVID-3324.png,COVID,COVID/images/COVID-3324.png,COVID/masks/COVID-3324.png,True,G02503,val,2aed22215a63b6fe77d31a0d5e850c9e,891d061a5f7f7868
COVID-3325.png,COVID,COVID/images/COVID-3325.png,COVID/masks/COVID-3325.png,True,G02504,train,d9d530e850b594c95efda3b220e11b8a,d13f46465b19294f
COVID-3326.png,COVID,COVID/images/COVID-3326.png,COVID/masks/COVID-3326.png,True,G02505,train,993b58b7ed2d435ffe48246f3434d087,91681e101b5f797b
COVID-3327.png,COVID,COVID/images/COVID-3327.png,COVID/masks/COVID-3327.png,True,G02506,train,b86568c4028bb74411251c360a07bf87,dd6a534b4b4e0a2d
COVID-3328.png,COVID,COVID/images/COVID-3328.png,COVID/masks/COVID-3328.png,True,G02507,train,c4115133660a1f9aac99c3b98d2b58fe,887d07161a1d5f5d
COVID-3329.png,COVID,COVID/images/COVID-3329.png,COVID/masks/COVID-3329.png,True,G02508,val,12dbe39f0cd4b28291d1284cabcad3f2,9179065e1a691f79
COVID-333.png,COVID,COVID/images/COVID-333.png,COVID/masks/COVID-333.png,True,G02509,val,13db5dd53e527dd73e0dca8e95177813,c42f434f6c7e60f0
COVID-3330.png,COVID,COVID/images/COVID-3330.png,COVID/masks/COVID-3330.png,True,G02510,test,696ca49f20ed9260475c107bef047016,8179d70a124d5b7b
COVID-3331.png,COVID,COVID/images/COVID-3331.png,COVID/masks/COVID-3331.png,True,G02511,train,c9b65717265532f248c0fb2e70643936,897dc39e9e453a14
COVID-3332.png,COVID,COVID/images/COVID-3332.png,COVID/masks/COVID-3332.png,True,G02512,test,1e8efbe5522905386413289619ebddcd,81781e115d7978cf
COVID-3333.png,COVID,COVID/images/COVID-3333.png,COVID/masks/COVID-3333.png,True,G02513,test,74d34da366f48a12f9b0b14e128fe6ba,803c57018f4f3b5f
COVID-3334.png,COVID,COVID/images/COVID-3334.png,COVID/masks/COVID-3334.png,True,G02514,train,459a68c76e0826aa06be778e362f9b1e,d52ac2431f4c6f3c
COVID-3335.png,COVID,COVID/images/COVID-3335.png,COVID/masks/COVID-3335.png,True,G02515,train,0b094ecfc1b1810b2654080d5491d8a7,817d060a1f5d5f59
COVID-3336.png,COVID,COVID/images/COVID-3336.png,COVID/masks/COVID-3336.png,True,G02516,train,4d1596154f37420f286f3a950a47ecd8,817907c71f17960f
COVID-3337.png,COVID,COVID/images/COVID-3337.png,COVID/masks/COVID-3337.png,True,G02517,train,43ff313ea3adbe3f89bc7d95d8bee839,993906c71f1c4f1d
COVID-3338.png,COVID,COVID/images/COVID-3338.png,COVID/masks/COVID-3338.png,True,G02518,train,57cd06de67546738862a41acd4fa1b66,817b861e494f7a69
COVID-3339.png,COVID,COVID/images/COVID-3339.png,COVID/masks/COVID-3339.png,True,G02519,train,fb1f86af77ca042bad8ce5c3833793ab,c5744b0d2d7f3c30
COVID-334.png,COVID,COVID/images/COVID-334.png,COVID/masks/COVID-334.png,True,G00209,test,89a37083045f12683ecb995c3b32c3e9,d939d6d65a485859
COVID-3340.png,COVID,COVID/images/COVID-3340.png,COVID/masks/COVID-3340.png,True,G02520,train,515b1ddc8eb9a48ad5402fc10004b7d5,d42de2d3531d4a4b
COVID-3341.png,COVID,COVID/images/COVID-3341.png,COVID/masks/COVID-3341.png,True,G02521,train,dae2b704c8f119835e11439d3b71ff59,81795e161a1ff859
COVID-3342.png,COVID,COVID/images/COVID-3342.png,COVID/masks/COVID-3342.png,True,G02522,train,bc8cf9e31e39f3695a54d9fbd02ac2ef,d4d043b04cf76cd3
COVID-3343.png,COVID,COVID/images/COVID-3343.png,COVID/masks/COVID-3343.png,True,G02523,train,84d5ef09cd058f335a07843403aab9fb,cc0e0f0e2cbcfce1
COVID-3344.png,COVID,COVID/images/COVID-3344.png,COVID/masks/COVID-3344.png,True,G02524,train,c83596b2f264762f5158ad3857f1fcc3,897c865a4d5f584d
COVID-3345.png,COVID,COVID/images/COVID-3345.png,COVID/masks/COVID-3345.png,True,G02525,train,3d4fab934bb1bf6dfc7a0689c24af4af,817b47161a595f59
COVID-3346.png,COVID,COVID/images/COVID-3346.png,COVID/masks/COVID-3346.png,True,G02526,train,e0acd4754a0f1bd8e2cadb7e657ed507,d0bd871f10451d5f
COVID-3347.png,COVID,COVID/images/COVID-3347.png,COVID/masks/COVID-3347.png,True,G02527,val,645a9b9e5529ee0e6387b1e1b8c5a43f,c0bdc2571b504f3e
COVID-3348.png,COVID,COVID/images/COVID-3348.png,COVID/masks/COVID-3348.png,True,G02528,test,7e2cff7deaa0a30bb0462559c54fd866,c466c34dea6ee0e1
COVID-3349.png,COVID,COVID/images/COVID-3349.png,COVID/masks/COVID-3349.png,True,G02529,train,37da86d2a43ce38fa5ac276bc815bddd,897d061e5b5b5a58
COVID-335.png,COVID,COVID/images/COVID-335.png,COVID/masks/COVID-335.png,True,G02530,val,458fa26406805133d5395e5237f31459,c83955e716055e5d
COVID-3350.png,COVID,COVID/images/COVID-3350.png,COVID/masks/COVID-3350.png,True,G02531,train,161be832e9b408030f8ca567f46647bd,d42d53c36d1c6a69
COVID-3351.png,COVID,COVID/images/COVID-3351.png,COVID/masks/COVID-3351.png,True,G02532,train,c5e54614be97a935f1b4ee59865be53b,897b071a105f5f1d
COVID-3352.png,COVID,COVID/images/COVID-3352.png,COVID/masks/COVID-3352.png,True,G02533,val,5a9021fb6fc4c222b6f5990979ebb653,993b97121a5d591e
COVID-3353.png,COVID,COVID/images/COVID-3353.png,COVID/masks/COVID-3353.png,True,G02534,train,a71a489036b84a67411b7fc5dc019331,997a461a1d4f1b1d
COVID-3354.png,COVID,COVID/images/COVID-3354.png,COVID/masks/COVID-3354.png,True,G02535,test,1990179175889353fae2711acc11918a,8129e6d558c66ed6
COVID-3355.png,COVID,COVID/images/COVID-3355.png,COVID/masks/COVID-3355.png,True,G02536,test,811bdf0b63416131673eed6b5a5d3a14,d42f461e496c3cf8
COVID-3356.png,COVID,COVID/images/COVID-3356.png,COVID/masks/COVID-3356.png,True,G02537,train,8f4f3ff885a0ce2485e9ac2c6dc98e54,d43d42c7d7125c4e
COVID-3357.png,COVID,COVID/images/COVID-3357.png,COVID/masks/COVID-3357.png,True,G02538,train,20a27aee7d6e401f042dc805aa5370a2,81787ec6d61a584f
COVID-3358.png,COVID,COVID/images/COVID-3358.png,COVID/masks/COVID-3358.png,True,G02539,train,4bfb969e85ff5a817532e43d41057c0a,913d4e19195f6879
COVID-3359.png,COVID,COVID/images/COVID-3359.png,COVID/masks/COVID-3359.png,True,G02540,train,9102d7f39c7631469606bc169270dcad,817906861e5fdd1e
COVID-336.png,COVID,COVID/images/COVID-336.png,COVID/masks/COVID-336.png,True,G02541,train,1419daf6f690badb7d55bc8b7bab5370,817806161a5f7f79
COVID-3360.png,COVID,COVID/images/COVID-3360.png,COVID/masks/COVID-3360.png,True,G02542,train,d1ef2eeb28cdac27218bca5af217760b,913d461a5a6d7969
COVID-3361.png,COVID,COVID/images/COVID-3361.png,COVID/masks/COVID-3361.png,True,G02543,train,8819dca5853ca3bb05640405243e3112,d42c0f5a5b6cb869
COVID-3362.png,COVID,COVID/images/COVID-3362.png,COVID/masks/COVID-3362.png,True,G02544,train,e819ddd2671de293cb11f61f1f302e4e,d52e42d71f444b1b
COVID-3363.png,COVID,COVID/images/COVID-3363.png,COVID/masks/COVID-3363.png,True,G02545,train,426dd3410c28e8bc205712bed8873953,98798716050e1fdf
COVID-3364.png,COVID,COVID/images/COVID-3364.png,COVID/masks/COVID-3364.png,True,G02546,train,c2dfe20aa9a71ba246dd26ab14e32fc6,dd2fd2434f0e2929
COVID-3365.png,COVID,COVID/images/COVID-3365.png,COVID/masks/COVID-3365.png,True,G02547,train,3ef26212ef2e8745bcf25b9c8affd194,dc2ed2d34b0f3a50
COVID-3366.png,COVID,COVID/images/COVID-3366.png,COVID/masks/COVID-3366.png,True,G02548,train,9dee09b7aeb1519317ce11d5742520e9,9168961353565f5b
COVID-3367.png,COVID,COVID/images/COVID-3367.png,COVID/masks/COVID-3367.png,True,G02549,val,82023fbfed427366dadc148c1dbd90f6,813d57125a1d793b
COVID-3368.png,COVID,COVID/images/COVID-3368.png,COVID/masks/COVID-3368.png,True,G02550,val,4f71d780527c76a343da080ef7faf149,817b07061a5d7b5b
COVID-3369.png,COVID,COVID/images/COVID-3369.png,COVID/masks/COVID-3369.png,True,G02551,val,9589b1bc07328cf50dabe48f88f2c781,d43d03431f5d074f
COVID-337.png,COVID,COVID/images/COVID-337.png,COVID/masks/COVID-337.png,True,G02552,test,6459110d270ec5b61bb3d97b771dd89b,913b475e585a445f
COVID-3370.png,COVID,COVID/images/COVID-3370.png,COVID/masks/COVID-3370.png,True,G02553,train,ac92e3f4c5c1ec8cb6bc41fe407d1fc2,c93bc71a184d5b1d
COVID-3371.png,COVID,COVID/images/COVID-3371.png,COVID/masks/COVID-3371.png,True,G02554,test,1ad7025fe285f4f7fa4ea494842dafc2,813b47c61e1c596f
COVID-3372.png,COVID,COVID/images/COVID-3372.png,COVID/masks/COVID-3372.png,True,G02555,train,3177ffdd6462104fb0a7fca12e2c41ea,d16e069a3ae53ab1
COVID-3373.png,COVID,COVID/images/COVID-3373.png,COVID/masks/COVID-3373.png,True,G02556,val,c5438ddcb2ef4d16bd0dae1b8e927b06,8179069e5e585d5d
COVID-3374.png,COVID,COVID/images/COVID-3374.png,COVID/masks/COVID-3374.png,True,G02556,val,c5438ddcb2ef4d16bd0dae1b8e927b06,8179069e5e585d5d
COVID-3375.png,COVID,COVID/images/COVID-3375.png,COVID/masks/COVID-3375.png,True,G02556,val,c5438ddcb2ef4d16bd0dae1b8e927b06,8179069e5e585d5d
COVID-3376.png,COVID,COVID/images/COVID-3376.png,COVID/masks/COVID-3376.png,True,G02556,val,c5438ddcb2ef4d16bd0dae1b8e927b06,8179069e5e585d5d
COVID-3377.png,COVID,COVID/images/COVID-3377.png,COVID/masks/COVID-3377.png,True,G02557,test,856a3ad172c183b3c18fc3137ca368e9,d42cc3570b4f2d3c
COVID-3378.png,COVID,COVID/images/COVID-3378.png,COVID/masks/COVID-3378.png,True,G02558,train,cb1fa78de4303d4f1c650f98de23290e,986d8796830f1f17
COVID-3379.png,COVID,COVID/images/COVID-3379.png,COVID/masks/COVID-3379.png,True,G02559,train,f8fdec133e33338e962720adf18fc9ef,d42e53d30b1f464b
COVID-338.png,COVID,COVID/images/COVID-338.png,COVID/masks/COVID-338.png,True,G02560,val,5794067284d6f9dd93e5e8d52e90e803,8978065a797979cc
COVID-3380.png,COVID,COVID/images/COVID-3380.png,COVID/masks/COVID-3380.png,True,G02561,train,1e057c0f63c57d19e4c37eff32b85b4b,817916965e1a595f
COVID-3381.png,COVID,COVID/images/COVID-3381.png,COVID/masks/COVID-3381.png,True,G02562,train,299c236e07df0824185cda9ae6087011,c17b07061e055f5f
COVID-3382.png,COVID,COVID/images/COVID-3382.png,COVID/masks/COVID-3382.png,True,G02563,train,e777262a1093267eec4fdfb742bd74af,817b161a5b39f9a1
COVID-3383.png,COVID,COVID/images/COVID-3383.png,COVID/masks/COVID-3383.png,True,G02564,train,b54ae6da7a3d3799fff3411ab27ffc82,817816161a5d5f7b
COVID-3384.png,COVID,COVID/images/COVID-3384.png,COVID/masks/COVID-3384.png,True,G02565,train,a711190521828801e5dd002d1f636615,d53d061b174e3971
COVID-3385.png,COVID,COVID/images/COVID-3385.png,COVID/masks/COVID-3385.png,True,G02566,train,1680582ad9b67a40d7fd9465fd3065e0,d02d461a1b4f7a79
COVID-3386.png,COVID,COVID/images/COVID-3386.png,COVID/masks/COVID-3386.png,True,G02567,train,04ba1429bfcbdcfdd2759b7402e4c107,892d861e1c5b5b4f
COVID-3387.png,COVID,COVID/images/COVID-3387.png,COVID/masks/COVID-3387.png,True,G02568,train,75bdbda7ab780bc2e1ca3b988591956a,d52f021a3b637666
COVID-3388.png,COVID,COVID/images/COVID-3388.png,COVID/masks/COVID-3388.png,True,G02569,val,db8b6328cadd052ebd33c6d7128ea0f1,99f9c71a934e1247
COVID-3389.png,COVID,COVID/images/COVID-3389.png,COVID/masks/COVID-3389.png,True,G02570,val,60093a10397087cf0109440c593659a7,d4a9c31fd04f3a58
COVID-339.png,COVID,COVID/images/COVID-339.png,COVID/masks/COVID-339.png,True,G02571,test,a4d54b44691865b8ab80b820c820c5e7,817d8796ba093d1d
COVID-3390.png,COVID,COVID/images/COVID-3390.png,COVID/masks/COVID-3390.png,True,G02572,test,62b44cf85a19542612bc0cb0f22c2c55,d85d47121f4f182f
COVID-3391.png,COVID,COVID/images/COVID-3391.png,COVID/masks/COVID-3391.png,True,G02573,test,2fc9e93d0d6866f27cc75b9cf51b20ad,91689e1a1f5b3959
COVID-3392.png,COVID,COVID/images/COVID-3392.png,COVID/masks/COVID-3392.png,True,G02574,train,c2be22202d279d52032ee051ee6e0e91,c42e33c36f3c2e2c
COVID-3393.png,COVID,COVID/images/COVID-3393.png,COVID/masks/COVID-3393.png,True,G02575,train,bd3a440cd674130a203bf7badcbf03e9,817b06865e1b785f
COVID-3394.png,COVID,COVID/images/COVID-3394.png,COVID/masks/COVID-3394.png,True,G02576,train,de7471b328ebda74c05dfbc8d673c45e,b868c79a4b4f584e
COVID-3395.png,COVID,COVID/images/COVID-3395.png,COVID/masks/COVID-3395.png,True,G02577,train,67a5cf0a2de8844c27226370ba57fea7,994d861a4d5f584f
COVID-3396.png,COVID,COVID/images/COVID-3396.png,COVID/masks/COVID-3396.png,True,G02578,train,4ea9b9a82779a5b22e1d6d45b69fc227,d0342f2d3d833bd2
COVID-3397.png,COVID,COVID/images/COVID-3397.png,COVID/masks/COVID-3397.png,True,G02579,train,a84e3ad511ec92dac9591c065a4ac930,dd2c420b59c74b5d
COVID-3398.png,COVID,COVID/images/COVID-3398.png,COVID/masks/COVID-3398.png,True,G02580,train,ab96c909cd24570355fe96355ced3bfb,d17b5743194d4b19
COVID-3399.png,COVID,COVID/images/COVID-3399.png,COVID/masks/COVID-3399.png,True,G02581,train,247fd1dc33a3cf000908d2df3effd31c,8179c6125b1d7b1d
COVID-34.png,COVID,COVID/images/COVID-34.png,COVID/masks/COVID-34.png,True,G02582,train,f7be0534e136b944752db324062133d4,cc6d961e6d141f0d
COVID-340.png,COVID,COVID/images/COVID-340.png,COVID/masks/COVID-340.png,True,G00465,train,7ffc478153d0540f5e5076f6dedf12e9,d93f561b444b194e
COVID-3400.png,COVID,COVID/images/COVID-3400.png,COVID/masks/COVID-3400.png,True,G02583,train,c1ce70eafebd605a93d2e81c8eff11ff,d03d4b0f1c3d7b28
COVID-3401.png,COVID,COVID/images/COVID-3401.png,COVID/masks/COVID-3401.png,True,G02584,train,d45cb0ef4dece2da3cdec2127a5ece3a,d52d02871f1d0b5f
COVID-3402.png,COVID,COVID/images/COVID-3402.png,COVID/masks/COVID-3402.png,True,G02585,train,9dcabb580b6198a675f8860c62a436e0,817b5a061a5b7d59
COVID-3403.png,COVID,COVID/images/COVID-3403.png,COVID/masks/COVID-3403.png,True,G02585,train,9dcabb580b6198a675f8860c62a436e0,817b5a061a5b7d59
COVID-3404.png,COVID,COVID/images/COVID-3404.png,COVID/masks/COVID-3404.png,True,G02586,train,c2a4c09e348046dae920c43e19715e09,817d4e165a5f581b
COVID-3405.png,COVID,COVID/images/COVID-3405.png,COVID/masks/COVID-3405.png,True,G02587,train,c9c8a98d23cc6707a40cdbd1d317c9d4,d46ec31b492e6c4d
COVID-3406.png,COVID,COVID/images/COVID-3406.png,COVID/masks/COVID-3406.png,True,G02588,val,e4c0a1ec16cb05b8bb2a654b7461f440,d0bde3d751420f91
COVID-3407.png,COVID,COVID/images/COVID-3407.png,COVID/masks/COVID-3407.png,True,G02589,test,c81aa23c444374b070230963582648e4,c17b16461e1b4d4f
COVID-3408.png,COVID,COVID/images/COVID-3408.png,COVID/masks/COVID-3408.png,True,G02590,test,016526304ce9d03dbadd786f4ce964c3,987b871e1a5d1b1a
COVID-3409.png,COVID,COVID/images/COVID-3409.png,COVID/masks/COVID-3409.png,True,G02591,val,5a0efcacdf759ca9de7942dc5fa22661,8068879ed6977ab1
COVID-341.png,COVID,COVID/images/COVID-341.png,COVID/masks/COVID-341.png,True,G02592,test,ee921088d691b311cd3b1bee357f6aa3,d42e424b5f4e2b1d
COVID-3410.png,COVID,COVID/images/COVID-3410.png,COVID/masks/COVID-3410.png,True,G02593,train,efe76f02bc83d87ebb9e3df3ca2c789c,997b071a174f1a59
COVID-3411.png,COVID,COVID/images/COVID-3411.png,COVID/masks/COVID-3411.png,True,G02594,train,6bc31323d96a09328d5a7bc561e20ad5,d267622965647772
COVID-3412.png,COVID,COVID/images/COVID-3412.png,COVID/masks/COVID-3412.png,True,G02595,train,bd332b0aba30cbef61d75a44db2648ca,817b161a1b3a5f3a
COVID-3413.png,COVID,COVID/images/COVID-3413.png,COVID/masks/COVID-3413.png,True,G02596,train,95285c05628f3a40fddfb72770ebe001,dc2f824f4d460f4e
COVID-3414.png,COVID,COVID/images/COVID-3414.png,COVID/masks/COVID-3414.png,True,G02597,val,b1bc22e3e5f9552496188781a9dbcdb6,dd2f821f070a5d56
COVID-3415.png,COVID,COVID/images/COVID-3415.png,COVID/masks/COVID-3415.png,True,G02598,train,f1294b62458e8674942192285dd96758,dc2ed30b4c2e1c6e
COVID-3416.png,COVID,COVID/images/COVID-3416.png,COVID/masks/COVID-3416.png,True,G02599,train,626808e5a2ba4057e4dc8782cb731bce,897b05071e1e475f
COVID-3417.png,COVID,COVID/images/COVID-3417.png,COVID/masks/COVID-3417.png,True,G02600,train,2043382e84f7e89301d5f290aad053fc,897b075e1a4d5b38
COVID-3418.png,COVID,COVID/images/COVID-3418.png,COVID/masks/COVID-3418.png,True,G02601,train,13f59d7af322795935a4a41d4df6b6a8,993b461e5a4b1af8
COVID-3419.png,COVID,COVID/images/COVID-3419.png,COVID/masks/COVID-3419.png,True,G02602,val,d05c06aca9d7742fb3054a8b4cbc026f,ccc253f01cd7acd2
COVID-342.png,COVID,COVID/images/COVID-342.png,COVID/masks/COVID-342.png,True,G02603,train,0f081094e8d02b8fcb46941b1d3f03ae,c129c716be4c7b38
COVID-3420.png,COVID,COVID/images/COVID-3420.png,COVID/masks/COVID-3420.png,True,G02604,train,6e66cd40f556700e85fd791f06a4a54f,d02dc3131ac1f73d
COVID-3421.png,COVID,COVID/images/COVID-3421.png,COVID/masks/COVID-3421.png,True,G02605,train,ece7c27761ceaebb34a6c26674da5e3a,817987161a1d5f5b
COVID-3422.png,COVID,COVID/images/COVID-3422.png,COVID/masks/COVID-3422.png,True,G02606,train,c3ad7662163b7e631b3097c4f184f405,917806069a1f7d5f
COVID-3423.png,COVID,COVID/images/COVID-3423.png,COVID/masks/COVID-3423.png,True,G02607,train,00452e4bb8679373672c0d8d4179b4ab,d12b57c64a585e4b
COVID-3424.png,COVID,COVID/images/COVID-3424.png,COVID/masks/COVID-3424.png,True,G02608,train,96a85d40c020052af95dd7cde811d5a1,dd3f06961a4f0d1c
COVID-3425.png,COVID,COVID/images/COVID-3425.png,COVID/masks/COVID-3425.png,True,G02609,train,34de7411cf1973c29ea3e30e0a0bdf49,d429d2434f5f4d49
COVID-3426.png,COVID,COVID/images/COVID-3426.png,COVID/masks/COVID-3426.png,True,G02610,train,c3fc7ea61ad8460554276112852d7375,81780746161f5f5f
COVID-3427.png,COVID,COVID/images/COVID-3427.png,COVID/masks/COVID-3427.png,True,G02610,train,c3fc7ea61ad8460554276112852d7375,81780746161f5f5f
COVID-3428.png,COVID,COVID/images/COVID-3428.png,COVID/masks/COVID-3428.png,True,G02611,val,958760da197ae6522df0afac5c3a1d08,9129061c5f7c7a5b
COVID-3429.png,COVID,COVID/images/COVID-3429.png,COVID/masks/COVID-3429.png,True,G02611,val,958760da197ae6522df0afac5c3a1d08,9129061c5f7c7a5b
COVID-343.png,COVID,COVID/images/COVID-343.png,COVID/masks/COVID-343.png,True,G02612,train,c158865eee627db98d6b2b4a6b890291,8079471e1e4f5d78
COVID-3430.png,COVID,COVID/images/COVID-3430.png,COVID/masks/COVID-3430.png,True,G02613,val,eb011daddc39f02758854d4fc2b926f6,c475a2dbc9c31d29
COVID-3431.png,COVID,COVID/images/COVID-3431.png,COVID/masks/COVID-3431.png,True,G02614,test,396b1521bfa1571e884cfb086e2ecd1a,d4ae830b45571d2f
COVID-3432.png,COVID,COVID/images/COVID-3432.png,COVID/masks/COVID-3432.png,True,G02615,test,1eb05b2d5396197a5a25043228a7c6ef,d03d43570b6e0b2f
COVID-3433.png,COVID,COVID/images/COVID-3433.png,COVID/masks/COVID-3433.png,True,G02616,train,c647463afa3a5a33f00dbdac5e69808d,81690f0597974f1f
COVID-3434.png,COVID,COVID/images/COVID-3434.png,COVID/masks/COVID-3434.png,True,G02617,train,175ee934a1a5a2e74dffe30b192d296d,817b160a5b5d6a79
COVID-3435.png,COVID,COVID/images/COVID-3435.png,COVID/masks/COVID-3435.png,True,G02618,train,1f39e3cb9a7add4e98554009e90a53a4,d939c54f1ac54b1a
COVID-3436.png,COVID,COVID/images/COVID-3436.png,COVID/masks/COVID-3436.png,True,G02619,test,2c0319485e9bec70a04f18f9b4813a12,d42e434d1cc62dbe
COVID-3437.png,COVID,COVID/images/COVID-3437.png,COVID/masks/COVID-3437.png,True,G02620,train,b0907c72f6879e5616eb4f42ca23fb32,81781e1855597b5f
COVID-3438.png,COVID,COVID/images/COVID-3438.png,COVID/masks/COVID-3438.png,True,G02621,train,b319016916aaa8961f5fa7da261867a1,993d871a1c4f5a5c
COVID-3439.png,COVID,COVID/images/COVID-3439.png,COVID/masks/COVID-3439.png,True,G02622,train,557b4270e8aa943aa68a515bccaafebe,857c061e596d6e6a
COVID-344.png,COVID,COVID/images/COVID-344.png,COVID/masks/COVID-344.png,True,G02623,train,88e568cf993d9df541ef4bae362956cf,d52ec2ff114d4d18
COVID-3440.png,COVID,COVID/images/COVID-3440.png,COVID/masks/COVID-3440.png,True,G02624,train,f9e6c4bca51915cd12e432c19c911b3a,997d87961f861a1a
COVID-3441.png,COVID,COVID/images/COVID-3441.png,COVID/masks/COVID-3441.png,True,G02625,train,7f45d6ba86fd5183890ffaa2bca6d78f,81ffd61845785559
COVID-3442.png,COVID,COVID/images/COVID-3442.png,COVID/masks/COVID-3442.png,True,G02626,train,7d7fc4302a553452447d9280f669a4dc,dc2cd3d30f4c4d1c
COVID-3443.png,COVID,COVID/images/COVID-3443.png,COVID/masks/COVID-3443.png,True,G02627,train,cf93f99a7ce1c9df439d53d40b7ef112,d02d5bd24b4f4f09
COVID-3444.png,COVID,COVID/images/COVID-3444.png,COVID/masks/COVID-3444.png,True,G02628,train,b975f0bd75070355430b7c28127df2bb,817987101f4f5a7b
COVID-3445.png,COVID,COVID/images/COVID-3445.png,COVID/masks/COVID-3445.png,True,G02629,val,e72b56859e42a49205af66c4f26fe9a0,c13e570a1b4ced4d
COVID-3446.png,COVID,COVID/images/COVID-3446.png,COVID/masks/COVID-3446.png,True,G02630,val,992921a92116900779f70a798aef2c89,ae2ad04d4d4a6f72
COVID-3447.png,COVID,COVID/images/COVID-3447.png,COVID/masks/COVID-3447.png,True,G02631,test,5b865bbd8c560f8ed77d7e8efaec503a,9c2f431c49593ce7
COVID-3448.png,COVID,COVID/images/COVID-3448.png,COVID/masks/COVID-3448.png,True,G02632,test,e519f9ba65b4196b9adc137561556158,fc2ec78b5b0f0a18
COVID-3449.png,COVID,COVID/images/COVID-3449.png,COVID/masks/COVID-3449.png,True,G02633,val,8e19e82a23ad852a047b20c5e0e47526,d42e43d70b4acf0b
COVID-345.png,COVID,COVID/images/COVID-345.png,COVID/masks/COVID-345.png,True,G02634,train,a8cf1666d0e172985be71604d7f9845d,dd2dc31b844f1d45
COVID-3450.png,COVID,COVID/images/COVID-3450.png,COVID/masks/COVID-3450.png,True,G02635,train,f8cc8a887f07d7045ad91e7df6e6e2c1,c42e03531f49dd6d
COVID-3451.png,COVID,COVID/images/COVID-3451.png,COVID/masks/COVID-3451.png,True,G02636,test,0b4fd19ae233761d1a2002951bb8aab4,d43d42d31b464f1e
COVID-3452.png,COVID,COVID/images/COVID-3452.png,COVID/masks/COVID-3452.png,True,G02637,train,c3c476520d768c77d8f7046e0419a3f6,817816871f5f4e1b
COVID-3453.png,COVID,COVID/images/COVID-3453.png,COVID/masks/COVID-3453.png,True,G02638,train,66d9a90d19c9b038b97a84e79215b188,9979d5d656c2521a
COVID-3454.png,COVID,COVID/images/COVID-3454.png,COVID/masks/COVID-3454.png,True,G02639,train,a34851655a7e24fbaf46f5206942088e,d42b43c75d4d1f18
COVID-3455.png,COVID,COVID/images/COVID-3455.png,COVID/masks/COVID-3455.png,True,G02640,train,8b366068ae631a8865e07917fb9d85f0,d52d429b0bc73bc4
COVID-3456.png,COVID,COVID/images/COVID-3456.png,COVID/masks/COVID-3456.png,True,G02641,train,f142b12c9ab97a3101842f57a0af931a,dd5d52525f69180e
COVID-3457.png,COVID,COVID/images/COVID-3457.png,COVID/masks/COVID-3457.png,True,G02642,train,5081e4b9403adc1d84b85d8db8f8d38e,9139861e5f5e385a
COVID-3458.png,COVID,COVID/images/COVID-3458.png,COVID/masks/COVID-3458.png,True,G02643,train,2d1070ddc4bbb21d8a540df6196adf8c,d43d034d4a6e2ddc
COVID-3459.png,COVID,COVID/images/COVID-3459.png,COVID/masks/COVID-3459.png,True,G02644,train,09c0310e494284820c74e77385e8d6eb,8169de9eb196061f
COVID-346.png,COVID,COVID/images/COVID-346.png,COVID/masks/COVID-346.png,True,G02645,train,69d54ee7f48dda8a071f006cf90e1077,cca6834b014fe56f
COVID-3460.png,COVID,COVID/images/COVID-3460.png,COVID/masks/COVID-3460.png,True,G02646,val,18e258d14767bae6877558cdc012ffa8,81b806975e873a9f
COVID-3461.png,COVID,COVID/images/COVID-3461.png,COVID/masks/COVID-3461.png,True,G02647,train,cc4992c4c930447d2763828b871e9969,917cc65a0f4f1b0d
COVID-3462.png,COVID,COVID/images/COVID-3462.png,COVID/masks/COVID-3462.png,True,G02648,train,8e607323b1be2d38494d6e4acc4a97d6,c43c074b5f653ce8
COVID-3463.png,COVID,COVID/images/COVID-3463.png,COVID/masks/COVID-3463.png,True,G02649,test,c2b891affa43cdb3e83d5d3b41696a71,985b861e5a595b5a
COVID-3464.png,COVID,COVID/images/COVID-3464.png,COVID/masks/COVID-3464.png,True,G02650,train,dbefbf7a3ad3a68a213f97aff42bc607,817816471a5edb7a
COVID-3465.png,COVID,COVID/images/COVID-3465.png,COVID/masks/COVID-3465.png,True,G02651,val,588e7b1cc0d1dd99f251a57140b9e1c0,d03d47461c0fed39
COVID-3466.png,COVID,COVID/images/COVID-3466.png,COVID/masks/COVID-3466.png,True,G02652,val,6e4d79f95fa2be3ce9c7005d64c299ee,d94c465a5d0d4b8f
COVID-3467.png,COVID,COVID/images/COVID-3467.png,COVID/masks/COVID-3467.png,True,G02653,test,4ab97354f08d75a745fdf4d40f38acb6,99e9c61e12475f51
COVID-3468.png,COVID,COVID/images/COVID-3468.png,COVID/masks/COVID-3468.png,True,G02654,test,300df7ed4e61929d3e343da6d7ad5f16,817f06125b79e879
COVID-3469.png,COVID,COVID/images/COVID-3469.png,COVID/masks/COVID-3469.png,True,G02655,train,e5bfb83b9671b25986a03d3414a82de9,d13ad7525a5a4939
COVID-347.png,COVID,COVID/images/COVID-347.png,COVID/masks/COVID-347.png,True,G02656,train,c78a8ddc7bc4b13b18c26ac55d9ec9ae,987d4596574e564a
COVID-3470.png,COVID,COVID/images/COVID-3470.png,COVID/masks/COVID-3470.png,True,G02657,train,d6ec28c12739d40d9117310842667346,d42cc30f5d444f5d
COVID-3471.png,COVID,COVID/images/COVID-3471.png,COVID/masks/COVID-3471.png,True,G02658,train,84284479effd5c5d0d0257b0f161d01d,dd2cc21f16c65f18
COVID-3472.png,COVID,COVID/images/COVID-3472.png,COVID/masks/COVID-3472.png,True,G02659,val,4972663328f7192f347cd8de871f3e68,d42f43430f4c6f2e
COVID-3473.png,COVID,COVID/images/COVID-3473.png,COVID/masks/COVID-3473.png,True,G02660,train,f11c7738477aed0b39ddd9526a49687d,91681e1b3aadb9a5
COVID-3474.png,COVID,COVID/images/COVID-3474.png,COVID/masks/COVID-3474.png,True,G02661,train,1fc4628f37c8caaf20b01fa9c6f8ed1a,d42d42535f0d6c6e
COVID-3475.png,COVID,COVID/images/COVID-3475.png,COVID/masks/COVID-3475.png,True,G02662,train,8d35d7cf94f30369fa82225825b381af,913806151fc74f5f
COVID-3476.png,COVID,COVID/images/COVID-3476.png,COVID/masks/COVID-3476.png,True,G02663,train,68cc2eb7b44768913654c5cc24a3a8d4,dd2cc75b194b1c4c
COVID-3477.png,COVID,COVID/images/COVID-3477.png,COVID/masks/COVID-3477.png,True,G02664,train,0614b7ce455e45accec35320c9c710ff,d42e03571f0d4d5d
COVID-3478.png,COVID,COVID/images/COVID-3478.png,COVID/masks/COVID-3478.png,True,G02665,train,67f4f6b6e7f6d9d1f190572ae9701245,887d85971e4f051f
COVID-3479.png,COVID,COVID/images/COVID-3479.png,COVID/masks/COVID-3479.png,True,G02666,test,9bf35963f82af59fae23f77f132a8b40,803957865f907f93
COVID-348.png,COVID,COVID/images/COVID-348.png,COVID/masks/COVID-348.png,True,G02667,train,ff67f31a678d1a9df67ff2d52d517d9c,9d2cc31b56471a6d
COVID-3480.png,COVID,COVID/images/COVID-3480.png,COVID/masks/COVID-3480.png,True,G02668,train,233f6c0224304ce4d43bc3a811bd8d5c,a96f061e186d68fc
COVID-3481.png,COVID,COVID/images/COVID-3481.png,COVID/masks/COVID-3481.png,True,G02669,test,1c3686074548f93805135fcedb79475b,917946561a0f6e3b
COVID-3482.png,COVID,COVID/images/COVID-3482.png,COVID/masks/COVID-3482.png,True,G02670,val,b21b01ef49643b4fdf58b8e363d654bc,817b96181f1a593f
COVID-3483.png,COVID,COVID/images/COVID-3483.png,COVID/masks/COVID-3483.png,True,G02671,train,e75d6ed4365ca7b34bcf475997ce325d,893d061a7d7a516b
COVID-3484.png,COVID,COVID/images/COVID-3484.png,COVID/masks/COVID-3484.png,True,G02672,train,7fee3fb3a88b813a25881223e4bc9d76,817a46461e5e4f5d
COVID-3485.png,COVID,COVID/images/COVID-3485.png,COVID/masks/COVID-3485.png,True,G02673,train,963cb7672a28a6ad5ea7e30309dab585,99684617575e1b4e
COVID-3486.png,COVID,COVID/images/COVID-3486.png,COVID/masks/COVID-3486.png,True,G02674,train,95d140244b577ceab7b3a5e56571d66b,893956571b964e1b
COVID-3487.png,COVID,COVID/images/COVID-3487.png,COVID/masks/COVID-3487.png,True,G02675,val,3c077ee5ed5b431d1704b7733b4145c6,dc3c534b4f4c0a2f
COVID-3488.png,COVID,COVID/images/COVID-3488.png,COVID/masks/COVID-3488.png,True,G02676,test,147af16441cea2817b8a09c65c43915b,897d86061a5d5f5a
COVID-3489.png,COVID,COVID/images/COVID-3489.png,COVID/masks/COVID-3489.png,True,G02676,test,147af16441cea2817b8a09c65c43915b,897d86061a5d5f5a
COVID-349.png,COVID,COVID/images/COVID-349.png,COVID/masks/COVID-349.png,True,G02677,train,fdec468965978ad4b97c8a6d8ec19369,8c7dd20f4827396d
COVID-3490.png,COVID,COVID/images/COVID-3490.png,COVID/masks/COVID-3490.png,True,G02678,train,41dd1fd523fe0caa6586bb592ff096d3,cc3cc3431b4a2f3b
COVID-3491.png,COVID,COVID/images/COVID-3491.png,COVID/masks/COVID-3491.png,True,G02679,train,a2e3aee13a029f1313033731c42e29e4,d46e9b1b4f4c2d0c
COVID-3492.png,COVID,COVID/images/COVID-3492.png,COVID/masks/COVID-3492.png,True,G02680,train,dd0fd7d2b2545e5fab7f12b7e6452663,817e061a5d4d795d
COVID-3493.png,COVID,COVID/images/COVID-3493.png,COVID/masks/COVID-3493.png,True,G02681,train,a252b8b1da3f2c24913f1fbd8b3c3ca2,d16b465b1a2d0d2f
COVID-3494.png,COVID,COVID/images/COVID-3494.png,COVID/masks/COVID-3494.png,True,G02682,train,8290e98f92c4046f4c36429b9ed73642,8b611c5f3a596879
COVID-3495.png,COVID,COVID/images/COVID-3495.png,COVID/masks/COVID-3495.png,True,G02683,train,6eb97ca92f90063c86547ec5d6b4f0c6,ce22d30bc14bcfc3
COVID-3496.png,COVID,COVID/images/COVID-3496.png,COVID/masks/COVID-3496.png,True,G02684,train,ccaa63f63a2df17428e3a13f37b4605a,d939571a1a591b39
COVID-3497.png,COVID,COVID/images/COVID-3497.png,COVID/masks/COVID-3497.png,True,G02685,train,68ee14fa1863e65b828854ed6ccb7b37,d4bc33c32ac62dc9
COVID-3498.png,COVID,COVID/images/COVID-3498.png,COVID/masks/COVID-3498.png,True,G02686,train,a19b05e9ca931e3d683588cfc43a827c,812d4edb1b4e6e1a
COVID-3499.png,COVID,COVID/images/COVID-3499.png,COVID/masks/COVID-3499.png,True,G02687,train,3698b763d2239443ed36443b0ddd0545,950e831b0d2d7e6d
COVID-35.png,COVID,COVID/images/COVID-35.png,COVID/masks/COVID-35.png,True,G01461,train,c916d1a103bbe063916ddc5e9e346ca7,9d2cc21b445f7c6c
COVID-350.png,COVID,COVID/images/COVID-350.png,COVID/masks/COVID-350.png,True,G02688,test,c1a8d22fdf28f793cdee1a085380ff95,fd2cc4939a830f1e
COVID-3500.png,COVID,COVID/images/COVID-3500.png,COVID/masks/COVID-3500.png,True,G02689,train,46803b86faf48bc1bed11e62f9c166ce,d43d03175b071b5b
COVID-3501.png,COVID,COVID/images/COVID-3501.png,COVID/masks/COVID-3501.png,True,G02690,val,90751bd97472ca3fec57d93f3e6b761c,c47d7f5a38286869
COVID-3502.png,COVID,COVID/images/COVID-3502.png,COVID/masks/COVID-3502.png,True,G02691,val,c77587cf3b7e4940a8a21447bb72af37,cd2c42535b4d4d5e
COVID-3503.png,COVID,COVID/images/COVID-3503.png,COVID/masks/COVID-3503.png,True,G02692,val,7e723433f91929377ec72348a43efec3,d56a47421b4b6d0f
COVID-3504.png,COVID,COVID/images/COVID-3504.png,COVID/masks/COVID-3504.png,True,G02693,test,15d32fc6302351e9eb0fe62bc31b7eb5,d42a9151455f5e6e
COVID-3505.png,COVID,COVID/images/COVID-3505.png,COVID/masks/COVID-3505.png,True,G02694,test,e93c75d3217f5ab4636facd72ef7135b,813f045e185d5f5b
COVID-3506.png,COVID,COVID/images/COVID-3506.png,COVID/masks/COVID-3506.png,True,G02695,test,8c7d1974e6097ef502fa19fe927405d7,d42f430d2cec3cdc
COVID-3507.png,COVID,COVID/images/COVID-3507.png,COVID/masks/COVID-3507.png,True,G02696,train,93f78cd42b2c0559cd5e1da96f541ad0,916886191d7d79ce
COVID-3508.png,COVID,COVID/images/COVID-3508.png,COVID/masks/COVID-3508.png,True,G02697,train,63a6b29e70ad5d39ed1c132d897d644e,8179861a5f781b5d
COVID-3509.png,COVID,COVID/images/COVID-3509.png,COVID/masks/COVID-3509.png,True,G02698,train,55ac132f18f0169168e9b7cb183983dc,913a474a1f4f0f1e
COVID-351.png,COVID,COVID/images/COVID-351.png,COVID/masks/COVID-351.png,True,G02699,train,12f661349cda0059365b258d4295cae4,d9a8e7179ac1c217
COVID-3510.png,COVID,COVID/images/COVID-3510.png,COVID/masks/COVID-3510.png,True,G02700,train,6741980dcebdc4857aa065f2ffd9b066,d92bc21a5d59385d
COVID-3511.png,COVID,COVID/images/COVID-3511.png,COVID/masks/COVID-3511.png,True,G02701,train,3e84651fcbd3b09564d2c223d2f4b8b1,d92946171a4d7e59
COVID-3512.png,COVID,COVID/images/COVID-3512.png,COVID/masks/COVID-3512.png,True,G02702,train,45a4e37dc365db9b3dd10a6b96002a1e,917f06065a696d7c
COVID-3513.png,COVID,COVID/images/COVID-3513.png,COVID/masks/COVID-3513.png,True,G02703,train,80c47ca2ce1f9fa92d64de3ad8cf8c1c,917f07161a4d4d7c
COVID-3514.png,COVID,COVID/images/COVID-3514.png,COVID/masks/COVID-3514.png,True,G02704,train,3baec3e632c51abbdfdc291d39bf32de,d47e46c79b1b4115
COVID-3515.png,COVID,COVID/images/COVID-3515.png,COVID/masks/COVID-3515.png,True,G02705,train,d72a483022bea7307051bb6d2ced0a3b,d02dc28397d64b97
COVID-3516.png,COVID,COVID/images/COVID-3516.png,COVID/masks/COVID-3516.png,True,G02705,train,d72a483022bea7307051bb6d2ced0a3b,d02dc28397d64b97
COVID-3517.png,COVID,COVID/images/COVID-3517.png,COVID/masks/COVID-3517.png,True,G02706,val,8d2a6ae2067e7e4d08ecda82ab457d8d,9929c61e451f1d4f
COVID-3518.png,COVID,COVID/images/COVID-3518.png,COVID/masks/COVID-3518.png,True,G02707,train,49529aa58abe1bb76e1e49cba5362361,d10f4e0af9e1e570
COVID-3519.png,COVID,COVID/images/COVID-3519.png,COVID/masks/COVID-3519.png,True,G02708,train,8b1c15908a58b6d0ee920279a89d099e,9169461a1bdb7999
COVID-352.png,COVID,COVID/images/COVID-352.png,COVID/masks/COVID-352.png,True,G02709,train,73a5007caa7f7e894fdee1045d4f3e7d,c80c431e1fe95aed
COVID-3520.png,COVID,COVID/images/COVID-3520.png,COVID/masks/COVID-3520.png,True,G02710,train,a6d730acd561eb3ae24c831c5133c8da,d42e83430f4eaf5c
COVID-3521.png,COVID,COVID/images/COVID-3521.png,COVID/masks/COVID-3521.png,True,G02711,val,92fa0bf50f14162bc8c5f96870710592,d46e035d0c6ea9d5
COVID-3522.png,COVID,COVID/images/COVID-3522.png,COVID/masks/COVID-3522.png,True,G02712,val,4daaf216f8f686a2b3974ae45c17a64f,91691e061f5f955a
COVID-3523.png,COVID,COVID/images/COVID-3523.png,COVID/masks/COVID-3523.png,True,G02713,test,892f87941ffa3aaf09c6c68b20943d15,917806161e5f3a5f
COVID-3524.png,COVID,COVID/images/COVID-3524.png,COVID/masks/COVID-3524.png,True,G02714,test,98a30979264e5c98cfd1e0b2ecc77968,dd3046561e4e3d5a
COVID-3525.png,COVID,COVID/images/COVID-3525.png,COVID/masks/COVID-3525.png,True,G02715,train,b4cb70d8044c0d9a8d4deb50375a3e8e,95bc865b13471d95
COVID-3526.png,COVID,COVID/images/COVID-3526.png,COVID/masks/COVID-3526.png,True,G02716,train,b096f623abbb4e63a0356d3201e09f8c,813d07471e1e9e5e
COVID-3527.png,COVID,COVID/images/COVID-3527.png,COVID/masks/COVID-3527.png,True,G02717,train,563bf42c9b2a40a265e9734ee0772ec2,986c871a1ecd155f
COVID-3528.png,COVID,COVID/images/COVID-3528.png,COVID/masks/COVID-3528.png,True,G02718,train,23d17508ac3436182d9866d53fe387ca,9d2ec25b19470e5d
COVID-3529.png,COVID,COVID/images/COVID-3529.png,COVID/masks/COVID-3529.png,True,G02719,train,7669b6c67f0702e93acd780e069c89fa,887d875e085d7879
COVID-353.png,COVID,COVID/images/COVID-353.png,COVID/masks/COVID-353.png,True,G02720,train,e4034813e09d43346279da841fe05a4c,8979d75e144b6958
COVID-3530.png,COVID,COVID/images/COVID-3530.png,COVID/masks/COVID-3530.png,True,G02721,test,402ca77aa2f9414649844bc11bea8b07,8178161e1b593b7b
COVID-3531.png,COVID,COVID/images/COVID-3531.png,COVID/masks/COVID-3531.png,True,G02722,train,ab08474191e073ae54d52268776ea002,913e465e584c6b79
COVID-3532.png,COVID,COVID/images/COVID-3532.png,COVID/masks/COVID-3532.png,True,G02723,train,54bbf733bcd8331cb8bf4043a51f8d12,957f87125d5d1a0c
COVID-3533.png,COVID,COVID/images/COVID-3533.png,COVID/masks/COVID-3533.png,True,G02724,train,c42baab30a2d9143d1440bd6190bf8e5,9919961e587b517a
COVID-3534.png,COVID,COVID/images/COVID-3534.png,COVID/masks/COVID-3534.png,True,G02725,train,da9a8cf3ad924b3adcb43ff1ed940131,9a13951e1a33bbb1
COVID-3535.png,COVID,COVID/images/COVID-3535.png,COVID/masks/COVID-3535.png,True,G02726,train,2cd886a283b162e37b202ecd551b7c18,c07d43570a4a6f79
COVID-3536.png,COVID,COVID/images/COVID-3536.png,COVID/masks/COVID-3536.png,True,G02727,train,e202900447d28e8ead3eeee42069d630,817bc554584b5b5b
COVID-3537.png,COVID,COVID/images/COVID-3537.png,COVID/masks/COVID-3537.png,True,G02728,train,8fe07bad81b7a1e8c331496e75ebd3de,d92946461e5d4b5b
COVID-3538.png,COVID,COVID/images/COVID-3538.png,COVID/masks/COVID-3538.png,True,G02729,val,0ac0690903c4f8ea11ebd0ae991ffa19,c17f84c6961a3d33
COVID-3539.png,COVID,COVID/images/COVID-3539.png,COVID/masks/COVID-3539.png,True,G02730,train,9b1cca60a168454ab44a50e3ff8d7096,816956161f5f7a38
COVID-354.png,COVID,COVID/images/COVID-354.png,COVID/masks/COVID-354.png,True,G02731,test,eb7aa436f1b8ffe232a3ec816158d0f8,fc6de2128b589c4d
COVID-3540.png,COVID,COVID/images/COVID-3540.png,COVID/masks/COVID-3540.png,True,G02732,val,28c06f14e6d264d81f5f4e23a868592a,816f4e4a2fada5a4
COVID-3541.png,COVID,COVID/images/COVID-3541.png,COVID/masks/COVID-3541.png,True,G02733,val,3dc756e5f87ed30fed728ac898f71fd4,913e9698abb9a5a4
COVID-3542.png,COVID,COVID/images/COVID-3542.png,COVID/masks/COVID-3542.png,True,G02734,test,feb144be4a599718406b39798591cae0,956ec71a454d194f
COVID-3543.png,COVID,COVID/images/COVID-3543.png,COVID/masks/COVID-3543.png,True,G02735,train,7508f5f82035c3e964c3827f82bada78,817b161a195d7b39
COVID-3544.png,COVID,COVID/images/COVID-3544.png,COVID/masks/COVID-3544.png,True,G02736,train,1e4fbf20396ef52d1514538d94aea88b,d43c4647191b4e6f
COVID-3545.png,COVID,COVID/images/COVID-3545.png,COVID/masks/COVID-3545.png,True,G02737,train,8aea855e4012028a55f0d7d3508e906d,817806061a5fcf7f
COVID-3546.png,COVID,COVID/images/COVID-3546.png,COVID/masks/COVID-3546.png,True,G02738,train,cf10208e46859126e22a88164b6e5bcf,d939461e1e4d5d38
COVID-3547.png,COVID,COVID/images/COVID-3547.png,COVID/masks/COVID-3547.png,True,G02739,train,94a41317b4ce738e8526f54f9d281fec,d56e42431b0f4f1b
COVID-3548.png,COVID,COVID/images/COVID-3548.png,COVID/masks/COVID-3548.png,True,G02739,train,4174f4449ec3f6c4c80a2aae15e69376,d56e42431b0f4f1b
COVID-3549.png,COVID,COVID/images/COVID-3549.png,COVID/masks/COVID-3549.png,True,G02740,train,74060a698222132ecbbad031f651fd8f,d52e46471b4d4f1a
COVID-355.png,COVID,COVID/images/COVID-355.png,COVID/masks/COVID-355.png,True,G00469,train,9ce4aab149975a6ba8ad274a874fcdb0,ed3d1796188907ba
COVID-3550.png,COVID,COVID/images/COVID-3550.png,COVID/masks/COVID-3550.png,True,G02741,train,ad36f6aee4e10d901b8f54ec6360f896,993f861a194d5b59
COVID-3551.png,COVID,COVID/images/COVID-3551.png,COVID/masks/COVID-3551.png,True,G02742,train,072f7e724c2fe2e37d6a3721806be5b6,917f04161a1b7b5b
COVID-3552.png,COVID,COVID/images/COVID-3552.png,COVID/masks/COVID-3552.png,True,G02743,train,aacb5cc6141d07623c0c7b4ec1d0726c,806cc71e194d6f5d
COVID-3553.png,COVID,COVID/images/COVID-3553.png,COVID/masks/COVID-3553.png,True,G02744,train,be1ceaf8de72a34fe7b16fae4d0deaec,81785e161e59595f
COVID-3554.png,COVID,COVID/images/COVID-3554.png,COVID/masks/COVID-3554.png,True,G02745,train,5d4f99d5e09ef619f22bddb27e0bc568,887f061a551b6b5b
COVID-3555.png,COVID,COVID/images/COVID-3555.png,COVID/masks/COVID-3555.png,True,G02746,train,e052e5c771113b163bb5892625a775f2,987d87161a1e535b
COVID-3556.png,COVID,COVID/images/COVID-3556.png,COVID/masks/COVID-3556.png,True,G02747,train,ab69a600728cf4c72cbcfd630a7df04e,dd2a531f5c1818eb
COVID-3557.png,COVID,COVID/images/COVID-3557.png,COVID/masks/COVID-3557.png,True,G02748,val,65565a672b1b751ab80cf7879b8d117f,d47f532f4c4c7c08
COVID-3558.png,COVID,COVID/images/COVID-3558.png,COVID/masks/COVID-3558.png,True,G02749,train,c57c0add7da90b1df1578e23719331ca,9d2e830b4d7c687c
COVID-3559.png,COVID,COVID/images/COVID-3559.png,COVID/masks/COVID-3559.png,True,G02750,val,efd4f69f69f75a35dd59e84d24aeccef,812d069e19df5ed8
COVID-356.png,COVID,COVID/images/COVID-356.png,COVID/masks/COVID-356.png,True,G02751,val,46f744244206a110feafe8cfb1a94552,fc32532e292d3535
COVID-3560.png,COVID,COVID/images/COVID-3560.png,COVID/masks/COVID-3560.png,True,G02752,test,44d37a45900257e9f0644e7c424c789c,95a52c967299699d
COVID-3561.png,COVID,COVID/images/COVID-3561.png,COVID/masks/COVID-3561.png,True,G02753,test,f39da6bc4c7550c7e396fed0d5658b93,8969871e5f5e1a4a
COVID-3562.png,COVID,COVID/images/COVID-3562.png,COVID/masks/COVID-3562.png,True,G02754,train,523f625ead35ae81d59c2edf1bc60975,817a5e9d69a56585
COVID-3563.png,COVID,COVID/images/COVID-3563.png,COVID/masks/COVID-3563.png,True,G02755,test,c1faa2d17e32ecb76297b979eff19c50,817b46161a7b6d59
COVID-3564.png,COVID,COVID/images/COVID-3564.png,COVID/masks/COVID-3564.png,True,G02756,train,41bcba060a7a9fa19032ce61bc324877,817b1e0e1a395d6b
COVID-3565.png,COVID,COVID/images/COVID-3565.png,COVID/masks/COVID-3565.png,True,G02757,train,5cefd8f77ed2d0e4450a8232f9598615,917846421b1fec6f
COVID-3566.png,COVID,COVID/images/COVID-3566.png,COVID/masks/COVID-3566.png,True,G02758,train,5604314c6a102129565a7c0660175e6e,817857461a5f4c6f
COVID-3567.png,COVID,COVID/images/COVID-3567.png,COVID/masks/COVID-3567.png,True,G02759,train,9db2842560382c691550d2e049e76ff7,817b94587beae1a1
COVID-3568.png,COVID,COVID/images/COVID-3568.png,COVID/masks/COVID-3568.png,True,G02760,val,9bc503ac3b45ff114aa8976f7ab5af7c,d92d861a155b1e5d
COVID-3569.png,COVID,COVID/images/COVID-3569.png,COVID/masks/COVID-3569.png,True,G02761,train,5787b7d9622d6e245e8a4f654c812beb,917c465a194d3a6f
COVID-357.png,COVID,COVID/images/COVID-357.png,COVID/masks/COVID-357.png,True,G02762,train,1811f31b582f8fde06dc4aafccbca421,d42f021f1d6c7ce4
COVID-3570.png,COVID,COVID/images/COVID-3570.png,COVID/masks/COVID-3570.png,True,G02763,train,7e941029ef9dc3a5fc3e18c14bd4413e,817c0e171f5f283d
COVID-3571.png,COVID,COVID/images/COVID-3571.png,COVID/masks/COVID-3571.png,True,G02764,val,86399a95f790bb94ae923be00f3cf32c,837b05145e545f1f
COVID-3572.png,COVID,COVID/images/COVID-3572.png,COVID/masks/COVID-3572.png,True,G02765,train,c61ef24e38b0863def44ed6f98aeafbe,95341e4a6a6a6b6b
COVID-3573.png,COVID,COVID/images/COVID-3573.png,COVID/masks/COVID-3573.png,True,G02766,train,b2a4a8615eb6bbc763cb5a58c50f4386,917f06180f195b5f
COVID-3574.png,COVID,COVID/images/COVID-3574.png,COVID/masks/COVID-3574.png,True,G02766,train,b2a4a8615eb6bbc763cb5a58c50f4386,917f06180f195b5f
COVID-3575.png,COVID,COVID/images/COVID-3575.png,COVID/masks/COVID-3575.png,True,G02767,train,8df277dfa6478c24ba298487e226e77a,d02f47431b2f2d2d
COVID-3576.png,COVID,COVID/images/COVID-3576.png,COVID/masks/COVID-3576.png,True,G02768,train,e50b368197525c1f690e5a2dbc8f90df,8829c75e1c4f5e1d
COVID-3577.png,COVID,COVID/images/COVID-3577.png,COVID/masks/COVID-3577.png,True,G02769,test,1a7e092e49893e6e7767e9a9c553530a,d13e06831b564f1f
COVID-3578.png,COVID,COVID/images/COVID-3578.png,COVID/masks/COVID-3578.png,True,G02770,train,0989fa845e9ba40c6a289619c36637a6,8c7dc31b4b471a4d
COVID-3579.png,COVID,COVID/images/COVID-3579.png,COVID/masks/COVID-3579.png,True,G02771,val,7f52331416839a72ecd6c8fda8337ae3,947c43d21ec70e5b
COVID-358.png,COVID,COVID/images/COVID-358.png,COVID/masks/COVID-358.png,True,G00271,test,e1685bbfee832287d849c75fbe62cef4,c947c35b5b415a4e
COVID-3580.png,COVID,COVID/images/COVID-3580.png,COVID/masks/COVID-3580.png,True,G02772,train,b89df5f2475ec4fd363ce8788d895844,c43e034b5d4f785c
COVID-3581.png,COVID,COVID/images/COVID-3581.png,COVID/masks/COVID-3581.png,True,G02773,val,08e5868a81cd5e8bd79676e72f80f240,897b05165a18ed5f
COVID-3582.png,COVID,COVID/images/COVID-3582.png,COVID/masks/COVID-3582.png,True,G02774,test,a946b88001a62cb878cd05df2337342b,897f861a851ea85f
COVID-3583.png,COVID,COVID/images/COVID-3583.png,COVID/masks/COVID-3583.png,True,G02775,test,01891861c13577dab8cdd10652f6506f,d929c61717471f54
COVID-3584.png,COVID,COVID/images/COVID-3584.png,COVID/masks/COVID-3584.png,True,G02776,train,20316db6c04f0e9ee4543cf38b27b4b6,b979c7961e44c61a
COVID-3585.png,COVID,COVID/images/COVID-3585.png,COVID/masks/COVID-3585.png,True,G02777,train,d455017470a2efcc9c9b9ab6c736f425,d46a070b2d383df5
COVID-3586.png,COVID,COVID/images/COVID-3586.png,COVID/masks/COVID-3586.png,True,G02778,train,b7e9673ec51e023ffb0b9604c5816d24,c83bc65e1d481b5d
COVID-3587.png,COVID,COVID/images/COVID-3587.png,COVID/masks/COVID-3587.png,True,G02779,train,54fe0093dfffce1326b6f55694362d17,997d865a1b470f1c
COVID-3588.png,COVID,COVID/images/COVID-3588.png,COVID/masks/COVID-3588.png,True,G02780,train,3cbb67bb072a12c0566911f1838ec030,91298619974f1b9f
COVID-3589.png,COVID,COVID/images/COVID-3589.png,COVID/masks/COVID-3589.png,True,G02781,train,36ae7ab7eb41ee0509914730772989f4,8179871e1ad55f1a
COVID-359.png,COVID,COVID/images/COVID-359.png,COVID/masks/COVID-359.png,True,G02782,train,8779347c1dc7d3835a55667b3b07f358,d42b071c3c071f5f
COVID-3590.png,COVID,COVID/images/COVID-3590.png,COVID/masks/COVID-3590.png,True,G02783,train,9930b01a3eee1cb5cef2105eb56fd3e5,dc2dd30b0e1a2e3e
COVID-3591.png,COVID,COVID/images/COVID-3591.png,COVID/masks/COVID-3591.png,True,G02783,train,9930b01a3eee1cb5cef2105eb56fd3e5,dc2dd30b0e1a2e3e
COVID-3592.png,COVID,COVID/images/COVID-3592.png,COVID/masks/COVID-3592.png,True,G02784,train,1538401f9c69fdb05a3a69d9c54eb649,dd2ee0930b490f1f
COVID-3593.png,COVID,COVID/images/COVID-3593.png,COVID/masks/COVID-3593.png,True,G02785,train,8fe5726ca18ec04ab1529d4abf8126a5,d929c7561e450b1f
COVID-3594.png,COVID,COVID/images/COVID-3594.png,COVID/masks/COVID-3594.png,True,G02786,train,77f6c3557b335207684fc7dbd38830f6,d46ec35b4d642c6c
COVID-3595.png,COVID,COVID/images/COVID-3595.png,COVID/masks/COVID-3595.png,True,G02787,train,196aabab039e9ba2c7fe162346c568a5,a869c61e5a4d1f1e
COVID-3596.png,COVID,COVID/images/COVID-3596.png,COVID/masks/COVID-3596.png,True,G02788,train,76551ebdbc5aab9b52d24c91d0663e4e,d42c53c34f469b1b
COVID-3597.png,COVID,COVID/images/COVID-3597.png,COVID/masks/COVID-3597.png,True,G02789,train,52b0a428eb36388f3334b587ad6ca081,997d071a17461e5d
COVID-3598.png,COVID,COVID/images/COVID-3598.png,COVID/masks/COVID-3598.png,True,G02790,val,ff111211d450f12698dffb5cb7f9067f,c03d66c77a386c63
COVID-3599.png,COVID,COVID/images/COVID-3599.png,COVID/masks/COVID-3599.png,True,G02790,val,ff111211d450f12698dffb5cb7f9067f,c03d66c77a386c63
COVID-36.png,COVID,COVID/images/COVID-36.png,COVID/masks/COVID-36.png,True,G02791,val,e49f77845437e61080de219b859f827f,d42f82435b474f0f
COVID-360.png,COVID,COVID/images/COVID-360.png,COVID/masks/COVID-360.png,True,G02792,test,893c70c4364a1956a52eb4999c27c037,dc2f021d4c4c7cf3
COVID-3600.png,COVID,COVID/images/COVID-3600.png,COVID/masks/COVID-3600.png,True,G02793,test,67129dbaa6dc48b3dd3fe84e67b2f4c3,d43c434b1f4e2f19
COVID-3601.png,COVID,COVID/images/COVID-3601.png,COVID/masks/COVID-3601.png,True,G02794,test,bb08844f858bd907b4a2bff3babc3829,817b560e185f6d78
COVID-3602.png,COVID,COVID/images/COVID-3602.png,COVID/masks/COVID-3602.png,True,G02795,train,f695e0a6894cc2b5d1db40d873c6d5d5,8879871e1e0f5e5a
COVID-3603.png,COVID,COVID/images/COVID-3603.png,COVID/masks/COVID-3603.png,True,G02796,train,aefa1642343630c23c31148495f296ce,997f06161add4f18
COVID-3604.png,COVID,COVID/images/COVID-3604.png,COVID/masks/COVID-3604.png,True,G02797,train,94b786b2a6d5d3b9717822be7005f4eb,815d4c5e59eda0e6
COVID-3605.png,COVID,COVID/images/COVID-3605.png,COVID/masks/COVID-3605.png,True,G02798,train,4fa785ecfc6963fdc1e35b96acb62b2a,d43b4b5f524e1b44
COVID-3606.png,COVID,COVID/images/COVID-3606.png,COVID/masks/COVID-3606.png,True,G02799,train,c04e13cc6e1c124541ecf49274667804,d43e83434fc22f2e
COVID-3607.png,COVID,COVID/images/COVID-3607.png,COVID/masks/COVID-3607.png,True,G02800,train,7051b7c9e3ea278c242544368189f936,d42d454b1a6d4d7a
COVID-3608.png,COVID,COVID/images/COVID-3608.png,COVID/masks/COVID-3608.png,True,G02801,train,f505ef7e8acc673d5d7b341409b86faf,91781c7b79e1a5c4
COVID-3609.png,COVID,COVID/images/COVID-3609.png,COVID/masks/COVID-3609.png,True,G02802,val,93c72da9a1405981d4526e0d872cc1a1,887dc75e18495f1c
COVID-361.png,COVID,COVID/images/COVID-361.png,COVID/masks/COVID-361.png,True,G02803,train,fd35977987430f31822eba374b4ff60d,d939c7961c4b3958
COVID-3610.png,COVID,COVID/images/COVID-3610.png,COVID/masks/COVID-3610.png,True,G02804,train,910d7b29bf66f28af4ce02ac9f5f1d9b,817d4c5e7ae861e1
COVID-3611.png,COVID,COVID/images/COVID-3611.png,COVID/masks/COVID-3611.png,True,G02805,val,b1989571b32d22c90f11275356503731,dd2a434b0a6b5c6d
COVID-3612.png,COVID,COVID/images/COVID-3612.png,COVID/masks/COVID-3612.png,True,G02806,train,2a7f77bd7ea430c4618a3c74494ec61f,817b84161a1f5f5b
COVID-3613.png,COVID,COVID/images/COVID-3613.png,COVID/masks/COVID-3613.png,True,G02807,test,c34c4f8c85b300956d17e346a85ac4b0,dc2ad3cb0945a91f
COVID-3614.png,COVID,COVID/images/COVID-3614.png,COVID/masks/COVID-3614.png,True,G02808,test,fc092f11865d4a501f9697b599415be1,8179161e581f5b5b
COVID-3615.png,COVID,COVID/images/COVID-3615.png,COVID/masks/COVID-3615.png,True,G02809,test,d8afafea7269cd10ddbbed7e255f3124,95956b6a6b8c9662
COVID-3616.png,COVID,COVID/images/COVID-3616.png,COVID/masks/COVID-3616.png,True,G02810,train,144acd7136e367938a3601deeee0b68d,817986161a5f5f1b
COVID-362.png,COVID,COVID/images/COVID-362.png,COVID/masks/COVID-362.png,True,G02811,val,d263c1b2b5ce155fa0e2faf64b15699d,917e041b5f195a5e
COVID-363.png,COVID,COVID/images/COVID-363.png,COVID/masks/COVID-363.png,True,G02812,test,090283788af8d04464a39d4a35ac2707,d12b531f19a5bca2
COVID-364.png,COVID,COVID/images/COVID-364.png,COVID/masks/COVID-364.png,True,G02813,train,dc0fcd722a0721296fdc93398f95385b,997987861ac54f1e
COVID-365.png,COVID,COVID/images/COVID-365.png,COVID/masks/COVID-365.png,True,G02814,train,b5ca54ad909bee332064e10300e40721,8179efa79612c287
COVID-366.png,COVID,COVID/images/COVID-366.png,COVID/masks/COVID-366.png,True,G02815,train,c3628bdd72ffb3263976781565484f76,dc2f461d1c2c3cb6
COVID-367.png,COVID,COVID/images/COVID-367.png,COVID/masks/COVID-367.png,True,G02816,train,c5848dfccf9f1e38f7bfe85e92e4757c,9c7cc3939b411e1e
COVID-368.png,COVID,COVID/images/COVID-368.png,COVID/masks/COVID-368.png,True,G02817,val,ca7500222aec162817ede603844014f1,985b061e19595f5b
COVID-369.png,COVID,COVID/images/COVID-369.png,COVID/masks/COVID-369.png,True,G02818,train,445581cdd924cf1e38c31a5c1a992af2,996de492c11fc31e
COVID-37.png,COVID,COVID/images/COVID-37.png,COVID/masks/COVID-37.png,True,G02819,train,6204574668a283eff4588d2d114bd48e,85ecae96c2c7ce0a
COVID-370.png,COVID,COVID/images/COVID-370.png,COVID/masks/COVID-370.png,True,G02820,train,bfe313f793769ddb7927349d8edbb240,8c6d4017d3561f5b
COVID-371.png,COVID,COVID/images/COVID-371.png,COVID/masks/COVID-371.png,True,G02821,train,71e7b9b2279c6dc718db6809f1f90410,dc2dc71c0c431f4f
COVID-372.png,COVID,COVID/images/COVID-372.png,COVID/masks/COVID-372.png,True,G00135,train,e908ace55b3ec9ff29351e16f8789f75,d869c5165b595e49
COVID-373.png,COVID,COVID/images/COVID-373.png,COVID/masks/COVID-373.png,True,G02822,train,21740822774283788ccea87b4293725a,81680f567a5e7879
COVID-374.png,COVID,COVID/images/COVID-374.png,COVID/masks/COVID-374.png,True,G02823,train,1c3f1919a34f1172d068104b85911748,dd68d61f431e420f
COVID-375.png,COVID,COVID/images/COVID-375.png,COVID/masks/COVID-375.png,True,G02824,train,d0210c268654eda940b62ca445bac6e0,d42c435f0a4e7f38
COVID-376.png,COVID,COVID/images/COVID-376.png,COVID/masks/COVID-376.png,True,G01087,train,523362f6ce2ea8c1e716b051a1f431bf,d42ee0a2dbc2f712
COVID-377.png,COVID,COVID/images/COVID-377.png,COVID/masks/COVID-377.png,True,G00111,train,d6742f8973f78964328fa29d213734b8,982d8097834f1fd7
COVID-378.png,COVID,COVID/images/COVID-378.png,COVID/masks/COVID-378.png,True,G02825,train,4b7e22124f0188b204cd21db11f70ede,913943d71a5e6c4e
COVID-379.png,COVID,COVID/images/COVID-379.png,COVID/masks/COVID-379.png,True,G02826,train,80e233ce16491e4ce1f8c1802a2aa80e,986d075ec21f7a49
COVID-38.png,COVID,COVID/images/COVID-38.png,COVID/masks/COVID-38.png,True,G01494,train,5a5652289c0c0eaae5b85bf12b857f62,9d2dc3131f461b1e
COVID-380.png,COVID,COVID/images/COVID-380.png,COVID/masks/COVID-380.png,True,G02827,train,53c5d37f54f685a82bf971d111f98657,9968165b0d794e5e
COVID-381.png,COVID,COVID/images/COVID-381.png,COVID/masks/COVID-381.png,True,G02828,train,af277a7689d13ec583cc03eb0e5c78dc,df2da15ee2127a0c
COVID-382.png,COVID,COVID/images/COVID-382.png,COVID/masks/COVID-382.png,True,G02829,train,b85fbec54d9d3f328165b70175dc1653,897d861aca39784f
COVID-383.png,COVID,COVID/images/COVID-383.png,COVID/masks/COVID-383.png,True,G02830,train,5e4d2eb8d79e791ac9b5cefa1836ee4f,c825971ed61e7949
COVID-384.png,COVID,COVID/images/COVID-384.png,COVID/masks/COVID-384.png,True,G02831,val,9b1b5ad908dc363422bba4af16b735d6,985bd65e6a593813
COVID-385.png,COVID,COVID/images/COVID-385.png,COVID/masks/COVID-385.png,True,G02832,val,d91e806e7b8bf1047046333a44612ab9,cd24d15a831f1f1d
COVID-386.png,COVID,COVID/images/COVID-386.png,COVID/masks/COVID-386.png,True,G02833,test,1276001c5bcc6941eef425af448e4366,995212593939a73f
COVID-387.png,COVID,COVID/images/COVID-387.png,COVID/masks/COVID-387.png,True,G02834,test,45abb7f0b2a67d6e397716e4928fd52a,9c69c316d3535b43
COVID-388.png,COVID,COVID/images/COVID-388.png,COVID/masks/COVID-388.png,True,G02835,train,a76b1531f401317851bc835ed9f03a22,cc6ed35bc11f6a04
COVID-389.png,COVID,COVID/images/COVID-389.png,COVID/masks/COVID-389.png,True,G02836,train,5b06c51d73d36aff17d1ac6178fce701,9d69841ec31fe21b
COVID-39.png,COVID,COVID/images/COVID-39.png,COVID/masks/COVID-39.png,True,G02837,train,60eabc85bef8196c504852103cdc79bc,ce2d494e6f0b0f0e
COVID-390.png,COVID,COVID/images/COVID-390.png,COVID/masks/COVID-390.png,True,G02838,train,3970e086e4f3b5dd922570f5380966fd,837a941a5f7c9cc4
COVID-391.png,COVID,COVID/images/COVID-391.png,COVID/masks/COVID-391.png,True,G02839,train,e73b556e54d6a67db9f89f19e7fb0735,d979161a18192f7e
COVID-392.png,COVID,COVID/images/COVID-392.png,COVID/masks/COVID-392.png,True,G02840,train,29d08ff45b964f460cc8f6d4b595b1bf,997dc416d743125b
COVID-393.png,COVID,COVID/images/COVID-393.png,COVID/masks/COVID-393.png,True,G02841,train,adfff967aa1c1d7146d0d37465d31153,917b061859d7f2e2
COVID-394.png,COVID,COVID/images/COVID-394.png,COVID/masks/COVID-394.png,True,G02842,train,609a7ea8d0f3943e646235b948815d22,d86f961e091f4f42
COVID-395.png,COVID,COVID/images/COVID-395.png,COVID/masks/COVID-395.png,True,G02843,train,8a58314756e96a883c4880d41bf20854,d969c61e6d213965
COVID-396.png,COVID,COVID/images/COVID-396.png,COVID/masks/COVID-396.png,True,G02844,val,9ac17e169537609c687fa3309d0c4811,8179871e1a4f1f1b
COVID-397.png,COVID,COVID/images/COVID-397.png,COVID/masks/COVID-397.png,True,G02845,train,b7afc6036ac24708dd5509763d9e95f9,9d6d8257831e0f1d
COVID-398.png,COVID,COVID/images/COVID-398.png,COVID/masks/COVID-398.png,True,G02846,train,b5f0403d1e12fddc49b0768ef4aff315,dd2e40431b454f3f
COVID-399.png,COVID,COVID/images/COVID-399.png,COVID/masks/COVID-399.png,True,G02847,test,a26f04d07e54b2af6e5f41f7f83c1a7f,917987863e5ece18
COVID-4.png,COVID,COVID/images/COVID-4.png,COVID/masks/COVID-4.png,True,G02848,test,023dd886818373be8dfbd50cb9579d05,91e1865ace6a963e
COVID-40.png,COVID,COVID/images/COVID-40.png,COVID/masks/COVID-40.png,True,G02849,train,8922cf8ebf45e8eefb8e4620bd1fa8f6,d979451f1c59491d
COVID-400.png,COVID,COVID/images/COVID-400.png,COVID/masks/COVID-400.png,True,G02850,train,1100cce69c7039c0af407f92620f799f,d929d61ec7421f49
COVID-401.png,COVID,COVID/images/COVID-401.png,COVID/masks/COVID-401.png,True,G02851,train,7276d821f2436756efebf6713a5b4f20,9c6f865b831c1c4f
COVID-402.png,COVID,COVID/images/COVID-402.png,COVID/masks/COVID-402.png,True,G02852,train,f2929e4d0d264193cb48a6529d582fc6,817bc6c6d8db191a
COVID-403.png,COVID,COVID/images/COVID-403.png,COVID/masks/COVID-403.png,True,G02853,val,2d555605c6977a09c9c149b90d189e40,916bc796d6c69612
COVID-404.png,COVID,COVID/images/COVID-404.png,COVID/masks/COVID-404.png,True,G02854,test,f40472038a4c7ce7345d65802e0e4d07,987f87120f071f0f
COVID-405.png,COVID,COVID/images/COVID-405.png,COVID/masks/COVID-405.png,True,G02855,train,3249fb4a70850bf58572053bea9cd43a,982f861e4b5b4b59
COVID-406.png,COVID,COVID/images/COVID-406.png,COVID/masks/COVID-406.png,True,G02856,val,4ca5a2d735e4e64d9a1545aba1b8bd70,817b8714125f5f1b
COVID-407.png,COVID,COVID/images/COVID-407.png,COVID/masks/COVID-407.png,True,G02857,train,cf6f4cd9c84e02426afa71c2837159b4,cc19530f0d292dbf
COVID-408.png,COVID,COVID/images/COVID-408.png,COVID/masks/COVID-408.png,True,G02858,train,fe3a6bd142fd57ce462e80e499fdcac3,9c6e821b8d1a9bc7
COVID-409.png,COVID,COVID/images/COVID-409.png,COVID/masks/COVID-409.png,True,G02859,train,0401a090be86900ac157f33016da04cf,d42b534f5e495a4a
COVID-41.png,COVID,COVID/images/COVID-41.png,COVID/masks/COVID-41.png,True,G00290,train,8479662f7e550eb62e30a973619489ed,9d2cc31b46471b6d
COVID-410.png,COVID,COVID/images/COVID-410.png,COVID/masks/COVID-410.png,True,G00489,train,968d6b18cee682c26de61856d260a018,fd2cc7130f451c4d
COVID-411.png,COVID,COVID/images/COVID-411.png,COVID/masks/COVID-411.png,True,G02860,train,4d62be47d8b8cd8caa1d502412d994ae,816e87194d3e7c4d
COVID-412.png,COVID,COVID/images/COVID-412.png,COVID/masks/COVID-412.png,True,G02861,train,9027678ea233f337e2a9d557aa0fa817,c965859b83818f9f
COVID-413.png,COVID,COVID/images/COVID-413.png,COVID/masks/COVID-413.png,True,G02862,train,31fa2c7d45f2fd1e82867fba6da62251,917e87124f4f1c59
COVID-414.png,COVID,COVID/images/COVID-414.png,COVID/masks/COVID-414.png,True,G00388,train,7bbbeddc7f424bd9dda05d3dc6ebf420,d829b59687965a1e
COVID-415.png,COVID,COVID/images/COVID-415.png,COVID/masks/COVID-415.png,True,G02863,train,a78af2d880ffb8520e41b210b7742c94,987f05c616165f1e
COVID-416.png,COVID,COVID/images/COVID-416.png,COVID/masks/COVID-416.png,True,G02864,train,dbaf6813d5288be9ae44942227de650e,986d861a4d5e394f
COVID-417.png,COVID,COVID/images/COVID-417.png,COVID/masks/COVID-417.png,True,G00394,train,3f88427f00d32b999fbd628f3915a9ae,c83975021736567f
COVID-418.png,COVID,COVID/images/COVID-418.png,COVID/masks/COVID-418.png,True,G02865,val,c1bb869241df686d89b7960787c60302,8378141a5b795b5e
COVID-419.png,COVID,COVID/images/COVID-419.png,COVID/masks/COVID-419.png,True,G00397,test,c071529d04a1608a705ddc741322ef2e,d83166c24eda1fd8
COVID-42.png,COVID,COVID/images/COVID-42.png,COVID/masks/COVID-42.png,True,G02866,train,9388d0e7d2cf481d96ecf5677baaf89d,817f460a69752d2f
COVID-420.png,COVID,COVID/images/COVID-420.png,COVID/masks/COVID-420.png,True,G02867,train,741f9428a530bd7acae6576655f2d311,9929c71e107b1c7d
COVID-421.png,COVID,COVID/images/COVID-421.png,COVID/masks/COVID-421.png,True,G02868,train,be0481490f497f67d5e0074dcfa354a2,ce2b4b4c2a2ce1f9
COVID-422.png,COVID,COVID/images/COVID-422.png,COVID/masks/COVID-422.png,True,G00401,test,4d3d08c5b8177f2ec29a8f85a3c2b6e2,d93de7474e0a1839
COVID-423.png,COVID,COVID/images/COVID-423.png,COVID/masks/COVID-423.png,True,G00506,train,55e8a83c51faa1b91ec36eed65e472da,9c75c51ac32b3b07
COVID-424.png,COVID,COVID/images/COVID-424.png,COVID/masks/COVID-424.png,True,G02869,val,72d3685666d2a72b9eaa5b83075844ee,817f9619151f1d4d
COVID-425.png,COVID,COVID/images/COVID-425.png,COVID/masks/COVID-425.png,True,G02870,test,0f7612824263c060645334672ceb74a2,883f461a1f4b1f59
COVID-426.png,COVID,COVID/images/COVID-426.png,COVID/masks/COVID-426.png,True,G02871,test,fe241f52115e338a93222698a0be1a9e,d92986160717575f
COVID-427.png,COVID,COVID/images/COVID-427.png,COVID/masks/COVID-427.png,True,G02872,test,0628e2738ac46de00bad789bf8034478,997d871a594d1e45
COVID-428.png,COVID,COVID/images/COVID-428.png,COVID/masks/COVID-428.png,True,G02634,train,a48aafa0848167c39b99f17d242ea7cc,dd2dc31b844f1d45
COVID-429.png,COVID,COVID/images/COVID-429.png,COVID/masks/COVID-429.png,True,G02873,train,876865010b3f4bf9020dac888b968168,d13f1619550f1c4f
COVID-43.png,COVID,COVID/images/COVID-43.png,COVID/masks/COVID-43.png,True,G01661,val,f0b2440cdbf896947499fe56af311c04,9d2cc41bb2529f56
COVID-430.png,COVID,COVID/images/COVID-430.png,COVID/masks/COVID-430.png,True,G02874,train,e4eebbdf83579585bbf4facc6183a6bb,8139778506173fda
COVID-431.png,COVID,COVID/images/COVID-431.png,COVID/masks/COVID-431.png,True,G02875,val,ae37d72fbb2ca21a8dfe35c9c925d5f7,d93b16161e0e5a5d
COVID-432.png,COVID,COVID/images/COVID-432.png,COVID/masks/COVID-432.png,True,G02876,train,516f5c5993806f950a4a766ea4f1b5d4,9c6f434d0c2eece8
COVID-433.png,COVID,COVID/images/COVID-433.png,COVID/masks/COVID-433.png,True,G02877,val,c8afce66138da0d8ba050f132bf18569,994b461a491d1eef
COVID-434.png,COVID,COVID/images/COVID-434.png,COVID/masks/COVID-434.png,True,G02878,train,74196c683ffb930db801194788dc1867,dd2a54521b6d2d2d
COVID-435.png,COVID,COVID/images/COVID-435.png,COVID/masks/COVID-435.png,True,G02879,test,79987ba3ba44cf1e1bc6c18ef5d381c2,89e926861e86d77a
COVID-436.png,COVID,COVID/images/COVID-436.png,COVID/masks/COVID-436.png,True,G02880,train,347f8472e19b80fe4180c3fc53f858a5,d43d420b4f0f1f4d
COVID-437.png,COVID,COVID/images/COVID-437.png,COVID/masks/COVID-437.png,True,G02881,train,5ff193ddf92b84a4eccf9132a1b76e5b,803f869e5e917ccc
COVID-438.png,COVID,COVID/images/COVID-438.png,COVID/masks/COVID-438.png,True,G02882,train,5db45098051ce9acb19bc5a7bdf3627b,9857561a491f1e1f
COVID-439.png,COVID,COVID/images/COVID-439.png,COVID/masks/COVID-439.png,True,G00293,train,8d26eba178d59a33ca00b924a880b102,8c24d25bc51f5b4b
COVID-44.png,COVID,COVID/images/COVID-44.png,COVID/masks/COVID-44.png,True,G00331,train,b37fdd167799634bc8338dbf16524dc4,d82dc61ac51f194f
COVID-440.png,COVID,COVID/images/COVID-440.png,COVID/masks/COVID-440.png,True,G02883,train,24fa0195da4757a37cc4b832fee02ce6,a8665259653d25b7
COVID-441.png,COVID,COVID/images/COVID-441.png,COVID/masks/COVID-441.png,True,G02884,train,24c066bae3528ef6eddc7bf655008659,ca7bb5149e86165a
COVID-442.png,COVID,COVID/images/COVID-442.png,COVID/masks/COVID-442.png,True,G00296,train,1d07d91524ce450805aa869329bcae29,d8798587c71f034b
COVID-443.png,COVID,COVID/images/COVID-443.png,COVID/masks/COVID-443.png,True,G02885,train,718cd9fabac0e2d73528f22f1ef33f97,d42ec35b4f403acd
COVID-444.png,COVID,COVID/images/COVID-444.png,COVID/masks/COVID-444.png,True,G02886,train,7c598b809d25a46d30475353616e59e9,9909d71e1a515b5b
COVID-445.png,COVID,COVID/images/COVID-445.png,COVID/masks/COVID-445.png,True,G02887,train,d1fe740c2354a45916569ab1d0f530b2,81701e795b3b39a5
COVID-446.png,COVID,COVID/images/COVID-446.png,COVID/masks/COVID-446.png,True,G02888,train,ae7e9b121c6ecb956122eb3bf60dfdac,932adf00ff00963f
COVID-447.png,COVID,COVID/images/COVID-447.png,COVID/masks/COVID-447.png,True,G00306,train,b75065717183168c7b97026d0215631f,cc6cc012cf4f0f4f
COVID-448.png,COVID,COVID/images/COVID-448.png,COVID/masks/COVID-448.png,True,G02889,train,163509e7d7845d636aedb1231671c894,885c4a6decf5f4c0
COVID-449.png,COVID,COVID/images/COVID-449.png,COVID/masks/COVID-449.png,True,G02890,train,67bb59a4441d775c74bb2603586d9b78,9d20d30d445e597f
COVID-45.png,COVID,COVID/images/COVID-45.png,COVID/masks/COVID-45.png,True,G02891,val,26fdc2e494e7113b8227538e7256f47d,8178861673777574
COVID-450.png,COVID,COVID/images/COVID-450.png,COVID/masks/COVID-450.png,True,G00309,train,8967dd433804a2218d60ddc841eb7494,9958d61f5b1a1e1c
COVID-451.png,COVID,COVID/images/COVID-451.png,COVID/masks/COVID-451.png,True,G02892,train,2a2d93810ce6b98e859c0c548657f5ad,93619d569c5c725c
COVID-452.png,COVID,COVID/images/COVID-452.png,COVID/masks/COVID-452.png,True,G02893,val,f6a1a118ee4d8cec6f3f9e6136249592,c928f530f6c2ea6a
COVID-453.png,COVID,COVID/images/COVID-453.png,COVID/masks/COVID-453.png,True,G02894,test,7284b5ded4258402a21c2ad4544444c8,d1b90ec61ec86979
COVID-454.png,COVID,COVID/images/COVID-454.png,COVID/masks/COVID-454.png,True,G02895,train,7294ba5f5080a01d751d0a2c024667c0,9c690756c21f5e69
COVID-455.png,COVID,COVID/images/COVID-455.png,COVID/masks/COVID-455.png,True,G00316,val,8200c8cc64c506309991c470705e08f8,d96d8416c51e5b4e
COVID-456.png,COVID,COVID/images/COVID-456.png,COVID/masks/COVID-456.png,True,G02896,val,4ed5b2bac107c071b4ff61a1bd479ec9,83e1045eb39e5e35
COVID-457.png,COVID,COVID/images/COVID-457.png,COVID/masks/COVID-457.png,True,G02897,test,323e816aa658033b1f59e8e96106a8d1,cc24920fd31ad71f
COVID-458.png,COVID,COVID/images/COVID-458.png,COVID/masks/COVID-458.png,True,G00319,test,a051475ec87c40a6a3327a27511bf8e3,9861e71ee51c1b59
COVID-459.png,COVID,COVID/images/COVID-459.png,COVID/masks/COVID-459.png,True,G02898,train,beb6f7a7bd3043baa08232e648c7c50d,d07bd2d2584b8f0b
COVID-46.png,COVID,COVID/images/COVID-46.png,COVID/masks/COVID-46.png,True,G02899,train,c4c02520dccae5f8b4253bf5a6d26052,9c2ec4d31a87e1da
COVID-460.png,COVID,COVID/images/COVID-460.png,COVID/masks/COVID-460.png,True,G02900,train,8876ad59b571b6a2f188cb27717854c1,9c6d0756c21f5a69
COVID-461.png,COVID,COVID/images/COVID-461.png,COVID/masks/COVID-461.png,True,G00323,train,3880646355c8ec316d252b7739f4b8db,dc6c821a871e1f5e
COVID-462.png,COVID,COVID/images/COVID-462.png,COVID/masks/COVID-462.png,True,G02901,train,66b5c65e52c092777854ecc4929a0d5c,9879d61e015f585d
COVID-463.png,COVID,COVID/images/COVID-463.png,COVID/masks/COVID-463.png,True,G02667,train,a13b485f57297b8afc63ee5535f8878e,9d2cc31b56471a6d
COVID-464.png,COVID,COVID/images/COVID-464.png,COVID/masks/COVID-464.png,True,G00503,train,d8fed92e53a21de09bca245095e3a84c,d57dc61a478e4287
COVID-465.png,COVID,COVID/images/COVID-465.png,COVID/masks/COVID-465.png,True,G02902,train,56fb22ee2a0087c95330ad059862948c,856c563a1ad76a5a
COVID-466.png,COVID,COVID/images/COVID-466.png,COVID/masks/COVID-466.png,True,G02903,train,a8a93f419fa2a0c254d67e884cda92d7,dc2ee0a2dbe3e302
COVID-467.png,COVID,COVID/images/COVID-467.png,COVID/masks/COVID-467.png,True,G02904,train,755c13d467a21cf7a987551eb6b8e049,d17b06461b3d6d61
COVID-468.png,COVID,COVID/images/COVID-468.png,COVID/masks/COVID-468.png,True,G02905,train,5170fab91db0b77ddf8c010b5eaa7e62,817f571a1f4d481d
COVID-469.png,COVID,COVID/images/COVID-469.png,COVID/masks/COVID-469.png,True,G02846,train,f85ee8eb68993403eb081d9e127d26c1,dd2e40431b454f3f
COVID-47.png,COVID,COVID/images/COVID-47.png,COVID/masks/COVID-47.png,True,G02906,val,e1cf625408ddb6d50f36beee61c8b622,dd7b421d0d81b783
COVID-470.png,COVID,COVID/images/COVID-470.png,COVID/masks/COVID-470.png,True,G00391,train,580d3a1ee689025b8c956a97ac79a16c,d07b46c65b197d21
COVID-471.png,COVID,COVID/images/COVID-471.png,COVID/masks/COVID-471.png,True,G02907,train,31aa59a3ff9ea24a39b6341a58fef59e,9169865e51ce565b
COVID-472.png,COVID,COVID/images/COVID-472.png,COVID/masks/COVID-472.png,True,G02908,train,25cf4af98a63379af2e75b8715cc364c,95ee07d16b857885
COVID-473.png,COVID,COVID/images/COVID-473.png,COVID/masks/COVID-473.png,True,G00417,val,3ec3948746a04fa3a9b2bad17c8c86c0,d81d43c71e1ad30f
COVID-474.png,COVID,COVID/images/COVID-474.png,COVID/masks/COVID-474.png,True,G02909,train,d5df895aeafe45dfbc78cea33df2ddca,816916121ad7ee5f
COVID-475.png,COVID,COVID/images/COVID-475.png,COVID/masks/COVID-475.png,True,G00378,test,8fee7fb2c72cb6d2dd54bbf215abb0ec,d42e830f4c6eeae1
COVID-476.png,COVID,COVID/images/COVID-476.png,COVID/masks/COVID-476.png,True,G00446,train,38451729f132b310b820a5da4b425344,d52e42431b1f0f5b
COVID-477.png,COVID,COVID/images/COVID-477.png,COVID/masks/COVID-477.png,True,G02910,train,94bc5bd54165076dc6b97bb005edcec8,81780e5f7978e0ab
COVID-478.png,COVID,COVID/images/COVID-478.png,COVID/masks/COVID-478.png,True,G00374,train,d5e59f62cd527fdf1ab43ac4e5a3b186,c43cd34b1e6c1b3c
COVID-479.png,COVID,COVID/images/COVID-479.png,COVID/masks/COVID-479.png,True,G00472,val,63cd04e485008c1093f4a08fe4093fd3,905b4fda3be064b1
COVID-48.png,COVID,COVID/images/COVID-48.png,COVID/masks/COVID-48.png,True,G02911,val,d486dcec1d88a9522403c5e5eeab3c25,9969c7165f4d3a18
COVID-480.png,COVID,COVID/images/COVID-480.png,COVID/masks/COVID-480.png,True,G02912,test,9c709dd31aef3f72eb101efe93383771,817877045e5e7a5a
COVID-481.png,COVID,COVID/images/COVID-481.png,COVID/masks/COVID-481.png,True,G02913,test,873edbd06ea9a5a16369cb6a8bb3818b,c03c2fcb3c92c33d
COVID-482.png,COVID,COVID/images/COVID-482.png,COVID/masks/COVID-482.png,True,G02914,test,5a510356146387331a770ccb17379f98,817846ce7a997a99
COVID-483.png,COVID,COVID/images/COVID-483.png,COVID/masks/COVID-483.png,True,G00177,val,cfe990870684862a96b73b8087192efd,8d75d21a671e1b45
COVID-484.png,COVID,COVID/images/COVID-484.png,COVID/masks/COVID-484.png,True,G02915,train,6247fc10b0625a3a20be3a16cdb21041,d42cf7434f2c2c3c
COVID-485.png,COVID,COVID/images/COVID-485.png,COVID/masks/COVID-485.png,True,G02916,train,c7de00843995a64eda8bf2fa4647fac6,d078464f4f2d3c1d
COVID-486.png,COVID,COVID/images/COVID-486.png,COVID/masks/COVID-486.png,True,G00489,train,af2243698dc32521f455418485d82ef5,fd2cc7130f451c4d
COVID-487.png,COVID,COVID/images/COVID-487.png,COVID/masks/COVID-487.png,True,G00209,test,02c75a7e92e2410e9d814a03a67380d9,d939d6d65a485859
COVID-488.png,COVID,COVID/images/COVID-488.png,COVID/masks/COVID-488.png,True,G02917,train,4662d062d76b4fc96e522927edb53aa1,817cc6131fce1b99
COVID-489.png,COVID,COVID/images/COVID-489.png,COVID/masks/COVID-489.png,True,G00381,train,a4ca6a4f8dd8e09fe23aba3038089673,987b45471e524b3b
COVID-49.png,COVID,COVID/images/COVID-49.png,COVID/masks/COVID-49.png,True,G02918,train,e0e0a347b19729c214fd71221f8401ab,c17e06c7ba14117f
COVID-490.png,COVID,COVID/images/COVID-490.png,COVID/masks/COVID-490.png,True,G02919,train,73582ca5ef5d71dcfd600284a7124026,d42e470b0c6c7cfc
COVID-491.png,COVID,COVID/images/COVID-491.png,COVID/masks/COVID-491.png,True,G00384,train,cc9e947a1f1d3e5ed9e8abdf67fd2944,d42e035f0f0a2f4f
COVID-492.png,COVID,COVID/images/COVID-492.png,COVID/masks/COVID-492.png,True,G00966,train,51f9fdbd8443d1ceade0fd5b336bc267,9869e5163ee570e4
COVID-493.png,COVID,COVID/images/COVID-493.png,COVID/masks/COVID-493.png,True,G02920,train,72574301dd5089d06dee58d3ed059fbb,c0754bcb4a0b6f4e
COVID-494.png,COVID,COVID/images/COVID-494.png,COVID/masks/COVID-494.png,True,G00349,train,2b51e4d7f229a9bafe6f12ed04a381a1,cd2c1253434b1f6f
COVID-495.png,COVID,COVID/images/COVID-495.png,COVID/masks/COVID-495.png,True,G02921,train,ff3e5ee52a6d068e237d44c9ee086b02,d961c71ed2453e61
COVID-496.png,COVID,COVID/images/COVID-496.png,COVID/masks/COVID-496.png,True,G02922,train,732b567c10c4e461cf10ede52be3768e,817d564a59093d3f
COVID-497.png,COVID,COVID/images/COVID-497.png,COVID/masks/COVID-497.png,True,G02923,train,1010a049787002d785fe97b06d6b50c1,9d6f821b471e194d
COVID-498.png,COVID,COVID/images/COVID-498.png,COVID/masks/COVID-498.png,True,G02924,train,73f9f00bf39395a885eb03868cd4b934,d42c53d74b0ac74b
COVID-499.png,COVID,COVID/images/COVID-499.png,COVID/masks/COVID-499.png,True,G02925,val,a5925797d236fb702462f4b132f22f55,d52d830f1e2c3f0d
COVID-5.png,COVID,COVID/images/COVID-5.png,COVID/masks/COVID-5.png,True,G02926,val,671a649204be2bf32d1beeab230fd662,9d2f40531bd3433e
COVID-50.png,COVID,COVID/images/COVID-50.png,COVID/masks/COVID-50.png,True,G02927,val,f5208eafd23b90e1a7ac05d25d0d7293,9d2cc01717171f8f
COVID-500.png,COVID,COVID/images/COVID-500.png,COVID/masks/COVID-500.png,True,G02928,test,70113bbbddfdf54f3e2b4ff004b62ccc,817cce301e3b794d
COVID-501.png,COVID,COVID/images/COVID-501.png,COVID/masks/COVID-501.png,True,G00430,train,9362cc86c3cedc19439f7c3a0cb49c04,8c4f834b4d7d6968
COVID-502.png,COVID,COVID/images/COVID-502.png,COVID/masks/COVID-502.png,True,G02929,train,a9a82044702e060bb4425edf0fa2837a,9c2ed193cb451cc7
COVID-503.png,COVID,COVID/images/COVID-503.png,COVID/masks/COVID-503.png,True,G02930,train,04c8bbda789164ffcd02ca1455f70693,cc06434b4d4f4f7c
COVID-504.png,COVID,COVID/images/COVID-504.png,COVID/masks/COVID-504.png,True,G02931,test,fc6bce094d2af346d1ded11bdc433c71,9d68c71f490e584f
COVID-505.png,COVID,COVID/images/COVID-505.png,COVID/masks/COVID-505.png,True,G02932,train,19c3d2a4f739601abeb81e5b9bece4d6,fd1ad21991ada6a1
COVID-506.png,COVID,COVID/images/COVID-506.png,COVID/masks/COVID-506.png,True,G02933,train,49229ada086ce2c4f0f6230d13a60c35,9c52d61e4a1e1e1f
COVID-507.png,COVID,COVID/images/COVID-507.png,COVID/masks/COVID-507.png,True,G02934,test,8b7a9a31bf68f462d0824757b9e89e7c,902bd7869e0c5d5b
COVID-508.png,COVID,COVID/images/COVID-508.png,COVID/masks/COVID-508.png,True,G00462,train,9bf1527ae411ca1bcd10a45b0c20b023,d92dd65396c64714
COVID-509.png,COVID,COVID/images/COVID-509.png,COVID/masks/COVID-509.png,True,G02935,train,8e6bfbb4f4ba1bb32096432e3396ec04,8d2ef01e9b5c2c65
COVID-51.png,COVID,COVID/images/COVID-51.png,COVID/masks/COVID-51.png,True,G02936,train,7ae1cc5f8aec9c2e6635c63e6b549698,cd2ed09293d13e1d
COVID-510.png,COVID,COVID/images/COVID-510.png,COVID/masks/COVID-510.png,True,G02937,train,640fa9a31f5f3e6ae9be85a77bfeda39,dd2b17160e5d191b
COVID-511.png,COVID,COVID/images/COVID-511.png,COVID/masks/COVID-511.png,True,G00492,train,603146dad1d81270ac1b62a4723b3fe5,9879e51697c61745
COVID-512.png,COVID,COVID/images/COVID-512.png,COVID/masks/COVID-512.png,True,G02938,train,e347029f8d1651a021cbf2f7a3c04210,995b461a590d1eaf
COVID-513.png,COVID,COVID/images/COVID-513.png,COVID/masks/COVID-513.png,True,G02939,train,cc0ce3ee61223b453a7ab65e8e447bfd,d97bd61c78293930
COVID-514.png,COVID,COVID/images/COVID-514.png,COVID/masks/COVID-514.png,True,G02940,train,5e118c8d5de4661963746fbb72329cf4,b865c71ec319bd41
COVID-515.png,COVID,COVID/images/COVID-515.png,COVID/masks/COVID-515.png,True,G02941,train,e8ce2ea9a0b4d3dcdaad5a5041f1a096,dc2c5243474b1f4f
COVID-516.png,COVID,COVID/images/COVID-516.png,COVID/masks/COVID-516.png,True,G02942,train,0de463ede19ebbc2554b745ef9997ec4,8179d7869c5cce54
COVID-517.png,COVID,COVID/images/COVID-517.png,COVID/masks/COVID-517.png,True,G01241,train,cedca9c81f7f43849262556103c7b5d5,dc2ec34b0d8a0f2f
COVID-518.png,COVID,COVID/images/COVID-518.png,COVID/masks/COVID-518.png,True,G02943,train,0212d81f1f2ed495feee98b603c46605,c86da5d293d98687
COVID-519.png,COVID,COVID/images/COVID-519.png,COVID/masks/COVID-519.png,True,G02944,train,2f20a6057b823f8a6d9a45a3184f6cd6,897a56c24e595bd2
COVID-52.png,COVID,COVID/images/COVID-52.png,COVID/masks/COVID-52.png,True,G02918,train,e0e0a347b19729c214fd71221f8401ab,c17e06c7ba14117f
COVID-520.png,COVID,COVID/images/COVID-520.png,COVID/masks/COVID-520.png,True,G00223,train,8f95c29ccc15f1957a6d19386bc5a7eb,883dc61e1a4f5b3a
COVID-521.png,COVID,COVID/images/COVID-521.png,COVID/masks/COVID-521.png,True,G02945,train,46171cf63976ce845265899522c9e0bd,d42bc62a6d693965
COVID-522.png,COVID,COVID/images/COVID-522.png,COVID/masks/COVID-522.png,True,G00704,train,317e02378c1683c5ce550faa85487b51,cc2fd30d455c4f43
COVID-523.png,COVID,COVID/images/COVID-523.png,COVID/masks/COVID-523.png,True,G02946,train,2721c649ad255bb47a1b8a0b0383859b,99bbd6ca4643294b
COVID-524.png,COVID,COVID/images/COVID-524.png,COVID/masks/COVID-524.png,True,G02947,train,10dba9666139a5716e80d7e54e0fec59,89bbd64a4e41395d
COVID-525.png,COVID,COVID/images/COVID-525.png,COVID/masks/COVID-525.png,True,G02948,train,0e2c0880cb7bc7dd16cd9b44bd883c87,dd2fd61e451a4945
COVID-526.png,COVID,COVID/images/COVID-526.png,COVID/masks/COVID-526.png,True,G02949,val,e0137e6273c9c94f6ab6227fe566c62d,897ad61819abbcac
COVID-527.png,COVID,COVID/images/COVID-527.png,COVID/masks/COVID-527.png,True,G01165,train,831ece52deaefeccbc14c8b688b618f2,d939e51f85c01ee8
COVID-528.png,COVID,COVID/images/COVID-528.png,COVID/masks/COVID-528.png,True,G02950,val,3a58b562841ae1b0ae5f210ac2da6c2b,90397e67e4069e1e
COVID-529.png,COVID,COVID/images/COVID-529.png,COVID/masks/COVID-529.png,True,G01099,train,6222d4d2ccf407526d21e5b0cb481141,d979e51785c01ee8
COVID-53.png,COVID,COVID/images/COVID-53.png,COVID/masks/COVID-53.png,True,G02951,test,9696dce9632de088aa049958f95b300f,dc2a535f4a4c1c7c
COVID-530.png,COVID,COVID/images/COVID-530.png,COVID/masks/COVID-530.png,True,G02952,test,b972f70ff84f1802952aefa4556a5fb2,b9394586164f4f1e
COVID-531.png,COVID,COVID/images/COVID-531.png,COVID/masks/COVID-531.png,True,G02953,val,5f70b8ce8fc3e34339f2e51d2ca389df,817b2e0e1e1e1a5f
COVID-532.png,COVID,COVID/images/COVID-532.png,COVID/masks/COVID-532.png,True,G01385,test,fccf9ee9eaada5f6a26614c9d79473c5,8178141e7b3b2b3b
COVID-533.png,COVID,COVID/images/COVID-533.png,COVID/masks/COVID-533.png,True,G02954,test,af8ebde143f2d9f067a4fd479b3ebba0,986d075ec21f5a69
COVID-534.png,COVID,COVID/images/COVID-534.png,COVID/masks/COVID-534.png,True,G02955,val,502bd576c0a5e9f94bac5144fe8c207b,91780e101b5b7b7b
COVID-535.png,COVID,COVID/images/COVID-535.png,COVID/masks/COVID-535.png,True,G02832,val,821fdcf81aa6375312a87892d9f40a70,cd24d15a831f1f1d
COVID-536.png,COVID,COVID/images/COVID-536.png,COVID/masks/COVID-536.png,True,G02956,test,b3de442fc763f13e3bae7abdf83e563d,917b4716585d3939
COVID-537.png,COVID,COVID/images/COVID-537.png,COVID/masks/COVID-537.png,True,G02957,train,f79d133985d267b46a6d956a64a3c1d0,9c2dc75f084a2d6d
COVID-538.png,COVID,COVID/images/COVID-538.png,COVID/masks/COVID-538.png,True,G02828,train,819fe78133b1d09227da719b0c28e583,df2da15ee2127a0c
COVID-539.png,COVID,COVID/images/COVID-539.png,COVID/masks/COVID-539.png,True,G02958,train,07798fdfcbf8296aaee8ddef20924d8c,dc6a531f6d1c2c2c
COVID-54.png,COVID,COVID/images/COVID-54.png,COVID/masks/COVID-54.png,True,G00328,train,8bb0c169aba08f1653bfd7f3c09f2d18,d87dc516851e1e4b
COVID-540.png,COVID,COVID/images/COVID-540.png,COVID/masks/COVID-540.png,True,G00288,train,3a484e15a40c2bfd08dedb71b37d19eb,883f831b290d1f3f
COVID-541.png,COVID,COVID/images/COVID-541.png,COVID/masks/COVID-541.png,True,G02830,train,6add187a0a1c6b84b2f53d0a471dd441,c825971ed61e7949
COVID-542.png,COVID,COVID/images/COVID-542.png,COVID/masks/COVID-542.png,True,G02959,train,3d44c48ac197e33d46ca83d3b83566d2,9c2a031f4f1c786f
COVID-543.png,COVID,COVID/images/COVID-543.png,COVID/masks/COVID-543.png,True,G02836,train,882c1c0e38177ef5b65c882708b822a1,9d69841ec31fe21b
COVID-544.png,COVID,COVID/images/COVID-544.png,COVID/masks/COVID-544.png,True,G02960,train,ded6c20925eeb6c4a808dc7809b03d04,d42cf353494f2e68
COVID-545.png,COVID,COVID/images/COVID-545.png,COVID/masks/COVID-545.png,True,G02961,train,b263796ad3490f9b753ea79039edb248,9728a6c67979446b
COVID-546.png,COVID,COVID/images/COVID-546.png,COVID/masks/COVID-546.png,True,G02962,train,401113e15887dc538bb753a93ffe6b6d,cd4d521a1b4d2f47
COVID-547.png,COVID,COVID/images/COVID-547.png,COVID/masks/COVID-547.png,True,G02963,train,4168e165d3ea4758a5d2efc1a015f99b,996b861e8e1e196b
COVID-548.png,COVID,COVID/images/COVID-548.png,COVID/masks/COVID-548.png,True,G01472,train,6633dd65005837917e69febdc40d8b74,9179061e5f5a3c5a
COVID-549.png,COVID,COVID/images/COVID-549.png,COVID/masks/COVID-549.png,True,G02964,train,e282dd2dec50f1d752258f64400c173a,997f061a48797969
COVID-55.png,COVID,COVID/images/COVID-55.png,COVID/masks/COVID-55.png,True,G02965,train,7189bdcd7e957e936fb8e5101693e621,917f07921a5d4d4b
COVID-550.png,COVID,COVID/images/COVID-550.png,COVID/masks/COVID-550.png,True,G02966,val,9fbd9b53941d93990bf67835a8a4a0f0,997f861e5a5838e8
COVID-551.png,COVID,COVID/images/COVID-551.png,COVID/masks/COVID-551.png,True,G02967,train,bae2f7c3add3473db04df67af765e787,887d521ed929396b
COVID-552.png,COVID,COVID/images/COVID-552.png,COVID/masks/COVID-552.png,True,G02968,train,cd36090a2bd979b3bb3323ad3e6faea1,9169c61edac73cd0
COVID-553.png,COVID,COVID/images/COVID-553.png,COVID/masks/COVID-553.png,True,G02969,train,1eafa262d7a3ed51246d53e0335bb514,d03ff392ceca2909
COVID-554.png,COVID,COVID/images/COVID-554.png,COVID/masks/COVID-554.png,True,G02970,train,4638530666225cb96d9bc2e9ad11f8a7,d969c71e1e301ec6
COVID-555.png,COVID,COVID/images/COVID-555.png,COVID/masks/COVID-555.png,True,G02971,train,6a2ea9b6511e803a31b2ba048043e907,d92bc71e8f009f8a
COVID-556.png,COVID,COVID/images/COVID-556.png,COVID/masks/COVID-556.png,True,G02972,val,eb2c02e372d5ef9b5124740c4f965263,c034adcd3c7c647a
COVID-557.png,COVID,COVID/images/COVID-557.png,COVID/masks/COVID-557.png,True,G02973,test,4d4a54439a70f6d8176c9d1e7941ae20,d83bc51696c71e1c
COVID-558.png,COVID,COVID/images/COVID-558.png,COVID/masks/COVID-558.png,True,G02974,test,268351ec00a51db071e09df84b5c93cd,856c471e5a7b7868
COVID-559.png,COVID,COVID/images/COVID-559.png,COVID/masks/COVID-559.png,True,G02975,train,66debf08746c616943b40e3e0bce1121,987d435647471a5b
COVID-56.png,COVID,COVID/images/COVID-56.png,COVID/masks/COVID-56.png,True,G01241,train,37e70afe1250154c8e09737f67d124ae,dc2ec34b0d8a0f2f
COVID-560.png,COVID,COVID/images/COVID-560.png,COVID/masks/COVID-560.png,True,G02976,train,8805f1259c12bf4fd83a8f51c5b8650c,89396386961ede4e
COVID-561.png,COVID,COVID/images/COVID-561.png,COVID/masks/COVID-561.png,True,G00495,val,cac9f97e864e74af7cca714bb38e16c0,d86c861b1e5a3e6c
COVID-562.png,COVID,COVID/images/COVID-562.png,COVID/masks/COVID-562.png,True,G02977,train,331ec78c1229bff9fb60298441e9466f,dc48531a4f4f1f15
COVID-563.png,COVID,COVID/images/COVID-563.png,COVID/masks/COVID-563.png,True,G02978,train,2515c7b494228feb4c8591799d594ca3,943ec2120f6b3f3c
COVID-564.png,COVID,COVID/images/COVID-564.png,COVID/masks/COVID-564.png,True,G02979,train,17f1a348a40c2a42735d21fa1bcbaba0,c234e8a584979dbe
COVID-565.png,COVID,COVID/images/COVID-565.png,COVID/masks/COVID-565.png,True,G02980,train,52e439c9c12efcf315b1ef8e17c7190e,c96d9692cb823a2f
COVID-566.png,COVID,COVID/images/COVID-566.png,COVID/masks/COVID-566.png,True,G02981,train,bfeec07c0fbe7c94f1619394a0ef920d,dc2e435d4c68f8e2
COVID-567.png,COVID,COVID/images/COVID-567.png,COVID/masks/COVID-567.png,True,G00255,train,71bbf25fa872d7edd186b8f4b02559be,dd25d20fd20bd668
COVID-568.png,COVID,COVID/images/COVID-568.png,COVID/masks/COVID-568.png,True,G02982,train,7f58c47f3edb8eb40768e8d09867a47c,dd2e425f4d485d4a
COVID-569.png,COVID,COVID/images/COVID-569.png,COVID/masks/COVID-569.png,True,G00355,train,0669f25bb80d13d47db8acbc6f54399d,8835d20fd319577c
COVID-57.png,COVID,COVID/images/COVID-57.png,COVID/masks/COVID-57.png,True,G00088,train,5a35b1a4c913b6abcfd1e998896b7a18,8d65c21b431f4f4d
COVID-570.png,COVID,COVID/images/COVID-570.png,COVID/masks/COVID-570.png,True,G02983,train,3c1e7750ae7ab758e3570138e9be0268,9d6d8257831e831f
COVID-571.png,COVID,COVID/images/COVID-571.png,COVID/masks/COVID-571.png,True,G02984,train,ebe488308933422fb012d27d73eed9eb,9d2c871b974f1a45
COVID-572.png,COVID,COVID/images/COVID-572.png,COVID/masks/COVID-572.png,True,G01661,val,ec6f6b7f2937a73a41a7239add4aa4c7,9d2cc41bb2529f56
COVID-573.png,COVID,COVID/images/COVID-573.png,COVID/masks/COVID-573.png,True,G02985,train,752e06735ebdb71555a87456a0a7121d,d9298496c716978f
COVID-574.png,COVID,COVID/images/COVID-574.png,COVID/masks/COVID-574.png,True,G02986,train,9eff077ae787f37c2b17b60528db3cd4,837a1610337b7e78
COVID-575.png,COVID,COVID/images/COVID-575.png,COVID/masks/COVID-575.png,True,G02987,test,f21443a5586013ca9b6d5a4f20947ba3,d46e831bc9856cc7
COVID-576.png,COVID,COVID/images/COVID-576.png,COVID/masks/COVID-576.png,True,G02988,train,805b8f6e0bd349279c58b60c9bc13de7,c879b5c6c6c2d25a
COVID-577.png,COVID,COVID/images/COVID-577.png,COVID/masks/COVID-577.png,True,G02989,train,92076bc6840e5b6be0d9210c5d6c52de,817f0746be17185b
COVID-578.png,COVID,COVID/images/COVID-578.png,COVID/masks/COVID-578.png,True,G02990,val,5416db134f774e25f582eecfee269ef1,cd2c870b035f386f
COVID-579.png,COVID,COVID/images/COVID-579.png,COVID/masks/COVID-579.png,True,G02991,test,50e062741dd621d6a3d92a1e78a95265,d869c71eb01abd1a
COVID-58.png,COVID,COVID/images/COVID-58.png,COVID/masks/COVID-58.png,True,G02992,val,4a9b8a0c73532832e98975ae7188215e,817d17c68613b85f
COVID-580.png,COVID,COVID/images/COVID-580.png,COVID/masks/COVID-580.png,True,G02993,val,6760d2d0cef15eddaa59d7f7741074f3,8179061e1b595f5e
COVID-581.png,COVID,COVID/images/COVID-581.png,COVID/masks/COVID-581.png,True,G02994,test,2b6f9438a3ebaedc719a089ebe9a0b42,915fc65e18ca3ba1
COVID-582.png,COVID,COVID/images/COVID-582.png,COVID/masks/COVID-582.png,True,G02995,test,a2819d2fbb76f20f4136ebe0d5eed804,d97dc3861e1a0b0f
COVID-583.png,COVID,COVID/images/COVID-583.png,COVID/masks/COVID-583.png,True,G02996,train,cc295c0f2517485fa30f9aed9927a32a,f97b045614135b5e
COVID-584.png,COVID,COVID/images/COVID-584.png,COVID/masks/COVID-584.png,True,G02997,train,c9998c5a02a394a7d8b56c00da4f53cc,8076e38b4bc57d68
COVID-585.png,COVID,COVID/images/COVID-585.png,COVID/masks/COVID-585.png,True,G02998,train,5b3c203f826e9106b4853848160e6abb,d42e03432f2e6e7a
COVID-586.png,COVID,COVID/images/COVID-586.png,COVID/masks/COVID-586.png,True,G02999,train,b2a7cd971c42fbb63efbd8c4138b81e5,9929c5961acf1e1e
COVID-587.png,COVID,COVID/images/COVID-587.png,COVID/masks/COVID-587.png,True,G03000,train,17c37f4d86002303fc2ada64ed3d2f47,d42a530b0e2c7e7d
COVID-588.png,COVID,COVID/images/COVID-588.png,COVID/masks/COVID-588.png,True,G03001,train,04afb919e4278bbf9b04235fdc24dd7d,9c2c8313d75f130f
COVID-589.png,COVID,COVID/images/COVID-589.png,COVID/masks/COVID-589.png,True,G03002,train,420774c46afa4435626428c0bd692e08,9979c5d6d6425343
COVID-59.png,COVID,COVID/images/COVID-59.png,COVID/masks/COVID-59.png,True,G03003,train,5b8bd52a662328148b3b21a7dba38397,cc2d92d38b494f3c
COVID-590.png,COVID,COVID/images/COVID-590.png,COVID/masks/COVID-590.png,True,G03004,train,c9398da4354a9c5ceb259d058856a2af,d52803db58e9c3b3
COVID-591.png,COVID,COVID/images/COVID-591.png,COVID/masks/COVID-591.png,True,G00000,test,5adae78f6d6ce500776641ac63d14e53,d17b561a59191b1e
COVID-592.png,COVID,COVID/images/COVID-592.png,COVID/masks/COVID-592.png,True,G03005,train,05f1b1f12ccacb6408baa93de7228a32,dc0d84960b0f1f5f
COVID-593.png,COVID,COVID/images/COVID-593.png,COVID/masks/COVID-593.png,True,G03006,test,c078e8d466f66f114ff1390f38f22e26,8921061d93c77f97
COVID-594.png,COVID,COVID/images/COVID-594.png,COVID/masks/COVID-594.png,True,G03007,train,2ef64ac393a40a670719e88c7a469586,d81907161e5a797e
COVID-595.png,COVID,COVID/images/COVID-595.png,COVID/masks/COVID-595.png,True,G02823,train,158535d5bd6c964a490f89be5e14fdc8,dd68d61f431e420f
COVID-596.png,COVID,COVID/images/COVID-596.png,COVID/masks/COVID-596.png,True,G03008,train,2ea66a6ff047f9de019dfe690716af89,91e8465313c77f91
COVID-597.png,COVID,COVID/images/COVID-597.png,COVID/masks/COVID-597.png,True,G03009,train,cf37c8c57e83d9f7adc7a8c244747a75,c965839b811f075f
COVID-598.png,COVID,COVID/images/COVID-598.png,COVID/masks/COVID-598.png,True,G03010,val,f1d377b8ee90e2c504ae4529715d37ac,817b5218393b3db3
COVID-599.png,COVID,COVID/images/COVID-599.png,COVID/masks/COVID-599.png,True,G02911,val,48f016476749b597b229313295747ce9,9969c7165f4d3a18
COVID-6.png,COVID,COVID/images/COVID-6.png,COVID/masks/COVID-6.png,True,G03011,train,e241b3cd501964715bade5353630ad2e,952f47971c4d451d
COVID-60.png,COVID,COVID/images/COVID-60.png,COVID/masks/COVID-60.png,True,G00126,train,3c43f7e56e7db9f98e38b709408fb8b8,9c65c61ec15a4f5a
COVID-600.png,COVID,COVID/images/COVID-600.png,COVID/masks/COVID-600.png,True,G03012,val,82a84d56ef6bea69a6177a45d5b3c2e1,d829b58e83da1e96
COVID-601.png,COVID,COVID/images/COVID-601.png,COVID/masks/COVID-601.png,True,G03013,val,78e3fe40db70c976e4864314c5465303,9919161a59f9f8e1
COVID-602.png,COVID,COVID/images/COVID-602.png,COVID/masks/COVID-602.png,True,G03014,test,200ba85353400c0461a34faee762bc90,d42f434f6c6a6ce0
COVID-603.png,COVID,COVID/images/COVID-603.png,COVID/masks/COVID-603.png,True,G03015,train,3e999713cc9d5cef8e529aa6252506df,d879f1c02da5668e
COVID-604.png,COVID,COVID/images/COVID-604.png,COVID/masks/COVID-604.png,True,G03016,train,5b7e2e67dcf8fbcfab13bf31b804d416,cc4e4b1fecb2b281
COVID-605.png,COVID,COVID/images/COVID-605.png,COVID/masks/COVID-605.png,True,G03017,train,89bb50d28fab13c6ae29433dcb92bbf9,da31e046475a5f5a
COVID-606.png,COVID,COVID/images/COVID-606.png,COVID/masks/COVID-606.png,True,G03018,train,2e25482ba81ce86be583e79ea2b90143,817b56161b197b59
COVID-607.png,COVID,COVID/images/COVID-607.png,COVID/masks/COVID-607.png,True,G03019,train,df275daf345213eb0d9efcd47de79a9f,d938e547e646964c
COVID-608.png,COVID,COVID/images/COVID-608.png,COVID/masks/COVID-608.png,True,G03020,train,9fa9d0e824026210c55c32aad5246a95,d43c4a096e6d1d6e
COVID-609.png,COVID,COVID/images/COVID-609.png,COVID/masks/COVID-609.png,True,G03021,train,862568b72290957262322fa9c6c9af26,9c6d0716c21f7a69
COVID-61.png,COVID,COVID/images/COVID-61.png,COVID/masks/COVID-61.png,True,G03022,train,e96502aaec89790fa87e6d6a326d71a9,c1795257da1a4b0f
COVID-610.png,COVID,COVID/images/COVID-610.png,COVID/masks/COVID-610.png,True,G03023,val,9453bdd1620e377ec2c6d27bebd3a20e,917d421b582f6f2c
COVID-611.png,COVID,COVID/images/COVID-611.png,COVID/masks/COVID-611.png,True,G03024,test,a82f6dbcaaa5ad3ec307696775686731,991957165f191a4f
COVID-612.png,COVID,COVID/images/COVID-612.png,COVID/masks/COVID-612.png,True,G03025,train,fe8c081cabd51944ad4da8f26ed145f6,cd2cb01afad21bc9
COVID-613.png,COVID,COVID/images/COVID-613.png,COVID/masks/COVID-613.png,True,G03026,train,006f92c15ec31cfa36042982c0c0dddd,817f065a5b4f1e1c
COVID-614.png,COVID,COVID/images/COVID-614.png,COVID/masks/COVID-614.png,True,G00467,train,890eb53e5026635e8ff272a7dbd9a7f0,c92d551217564b5f
COVID-615.png,COVID,COVID/images/COVID-615.png,COVID/masks/COVID-615.png,True,G00554,train,0ebd284b77c99122e0c53cdc1619a869,c149165e0f6d2d3d
COVID-616.png,COVID,COVID/images/COVID-616.png,COVID/masks/COVID-616.png,True,G03027,test,3f393b114734dd5723059b38cf54d7dc,d42ec30f4f4b0a4f
COVID-617.png,COVID,COVID/images/COVID-617.png,COVID/masks/COVID-617.png,True,G03028,train,87fc890479208cab63512ee3569a4dac,cc6f42d24b4d3e4a
COVID-618.png,COVID,COVID/images/COVID-618.png,COVID/masks/COVID-618.png,True,G03029,train,47bb8def9b256302c505eb67aa14264e,d42a72939f40475f
COVID-619.png,COVID,COVID/images/COVID-619.png,COVID/masks/COVID-619.png,True,G00682,val,85d574829891927f42f7cf32b684a728,9b65c41ae60ebf42
COVID-62.png,COVID,COVID/images/COVID-62.png,COVID/masks/COVID-62.png,True,G03007,train,ada8cf0fd69aea3430afa7808e016f77,d81907161e5a797e
COVID-620.png,COVID,COVID/images/COVID-620.png,COVID/masks/COVID-620.png,True,G03030,train,487a44e16827bd847709937994f420eb,dd2e1283cbe03a97
COVID-621.png,COVID,COVID/images/COVID-621.png,COVID/masks/COVID-621.png,True,G03031,train,9057f6d45a97afc85e5f7b447c419541,8c3dd34b6d402d7c
COVID-622.png,COVID,COVID/images/COVID-622.png,COVID/masks/COVID-622.png,True,G03032,val,bfa0cb786a9afc79175a9c7a2769b2bf,da4985564b4b5e0f
COVID-623.png,COVID,COVID/images/COVID-623.png,COVID/masks/COVID-623.png,True,G03033,val,1bdd8a54482824f16a0e6bb63466dc96,d42a4f4c0a0fe9bb
COVID-624.png,COVID,COVID/images/COVID-624.png,COVID/masks/COVID-624.png,True,G03034,test,756f7e0d844debe1fdf4eb1030a717ae,9c36a20bc7170fc7
COVID-625.png,COVID,COVID/images/COVID-625.png,COVID/masks/COVID-625.png,True,G03035,train,2b5aeba29998309765ff6d69df33d7e0,db0915165e1a5ebc
COVID-626.png,COVID,COVID/images/COVID-626.png,COVID/masks/COVID-626.png,True,G03036,train,1aca92148e4c9dfe73bc822a83cccca2,d42ec10f6cea2e74
COVID-627.png,COVID,COVID/images/COVID-627.png,COVID/masks/COVID-627.png,True,G03037,train,94f8d2f04a349790fb735532048ca578,dda8d65ed4425f40
COVID-628.png,COVID,COVID/images/COVID-628.png,COVID/masks/COVID-628.png,True,G03038,train,9d2b2304ebcb12c05a39cd79d2c5183e,8d0d41d21acb5b5f
COVID-629.png,COVID,COVID/images/COVID-629.png,COVID/masks/COVID-629.png,True,G03039,train,a12a1b5c07acaf9b6304242326b03d59,837ed59551175417
COVID-63.png,COVID,COVID/images/COVID-63.png,COVID/masks/COVID-63.png,True,G00155,train,2faa9a8742833874b89f8c6a649db858,987fd71a5c435a18
COVID-630.png,COVID,COVID/images/COVID-630.png,COVID/masks/COVID-630.png,True,G03040,train,6be6af0e398fafcb8c468b4ef7f8c0f8,b86d025e495d595d
COVID-631.png,COVID,COVID/images/COVID-631.png,COVID/masks/COVID-631.png,True,G03041,train,d6eee2fb873b9709717c2c5f81712474,883f5d1c581939eb
COVID-632.png,COVID,COVID/images/COVID-632.png,COVID/masks/COVID-632.png,True,G03042,train,ac78876070d171a5f9b123ec6f5ebf47,d52b52471b1c4f59
COVID-633.png,COVID,COVID/images/COVID-633.png,COVID/masks/COVID-633.png,True,G00124,train,a6f3ba126623b133aa379ee3031cc65e,847e435a1f4b1d4d
COVID-634.png,COVID,COVID/images/COVID-634.png,COVID/masks/COVID-634.png,True,G03043,train,08ea66561911b9e4b5d7ebad06b6109c,887d00d616c75f5d
COVID-635.png,COVID,COVID/images/COVID-635.png,COVID/masks/COVID-635.png,True,G03044,val,b88ffca2513a860329583f6aea9af458,d52b431c6232e7d3
COVID-636.png,COVID,COVID/images/COVID-636.png,COVID/masks/COVID-636.png,True,G03045,train,e68e322eafe60a5c3a25ff68d8871da8,9f6954de494b4ac8
COVID-637.png,COVID,COVID/images/COVID-637.png,COVID/masks/COVID-637.png,True,G03046,val,a5bfd817c43b0b3ed878abe7dc8df833,817ac496389f5b93
COVID-638.png,COVID,COVID/images/COVID-638.png,COVID/masks/COVID-638.png,True,G03047,val,839923253dce090d1e33e44d5f683fb7,dd2a524f5d181f0e
COVID-639.png,COVID,COVID/images/COVID-639.png,COVID/masks/COVID-639.png,True,G03048,test,f246d9cab1c23dd310fdee7f215b54c7,d968059668694fee
COVID-64.png,COVID,COVID/images/COVID-64.png,COVID/masks/COVID-64.png,True,G03049,train,9ce0bfa53efc936e14c1d3c888595aef,d43fc25755433b18
COVID-640.png,COVID,COVID/images/COVID-640.png,COVID/masks/COVID-640.png,True,G03050,test,a8a97099b8b17a58f6455e3069ae0d4c,956807161e3f1d3d
COVID-641.png,COVID,COVID/images/COVID-641.png,COVID/masks/COVID-641.png,True,G02840,train,624e5fabfce6cb5d3922b01a63a67964,997dc416d743125b
COVID-642.png,COVID,COVID/images/COVID-642.png,COVID/masks/COVID-642.png,True,G03051,train,63697e908302e98a7b83bc40e32c7894,9c0643534f4d5f4b
COVID-643.png,COVID,COVID/images/COVID-643.png,COVID/masks/COVID-643.png,True,G03052,train,bdfc6695076c71ec5c67dcce231a9fe4,883dd2431dd75b19
COVID-644.png,COVID,COVID/images/COVID-644.png,COVID/masks/COVID-644.png,True,G02843,train,ea43f868d36801cbabe573ca4201e580,d969c61e6d213965
COVID-645.png,COVID,COVID/images/COVID-645.png,COVID/masks/COVID-645.png,True,G03053,train,af2b96bb79b921de6dd367f701196f9c,d95f561e1a0b2aac
COVID-646.png,COVID,COVID/images/COVID-646.png,COVID/masks/COVID-646.png,True,G03054,test,6e203120bb0a28278132664be67e3120,997b461b1e4b184f
COVID-647.png,COVID,COVID/images/COVID-647.png,COVID/masks/COVID-647.png,True,G03055,train,89825414c791cfc35462797634e04645,9979c5d69646074e
COVID-648.png,COVID,COVID/images/COVID-648.png,COVID/masks/COVID-648.png,True,G03056,train,f08ecf001af4232c815056747b63d9fb,d968061a596d7979
COVID-649.png,COVID,COVID/images/COVID-649.png,COVID/masks/COVID-649.png,True,G03057,train,814b3fb05be7bd7cb4a0ed537568f93a,c0bc2f832d4d5e4b
COVID-65.png,COVID,COVID/images/COVID-65.png,COVID/masks/COVID-65.png,True,G02867,train,00a4909c5ebf258b236fba00443fc369,9929c71e107b1c7d
COVID-650.png,COVID,COVID/images/COVID-650.png,COVID/masks/COVID-650.png,True,G03058,train,80730b323165cad9c4467a81dbafb83a,f87d47161e033c9c
COVID-651.png,COVID,COVID/images/COVID-651.png,COVID/masks/COVID-651.png,True,G03059,train,1c6bed45e51499e99f77bbc8a8cac601,9978161a5979e9e1
COVID-652.png,COVID,COVID/images/COVID-652.png,COVID/masks/COVID-652.png,True,G03060,train,b8b143080b6e518bc5607640d96de87d,d47f23922b4d4157
COVID-653.png,COVID,COVID/images/COVID-653.png,COVID/masks/COVID-653.png,True,G03061,train,23edb00e7bb24726213a4a66614948e9,986b67971ec2581e
COVID-654.png,COVID,COVID/images/COVID-654.png,COVID/masks/COVID-654.png,True,G03062,train,b13d8377dc91e99e5b9c8534184498dc,817b971a125f585b
COVID-655.png,COVID,COVID/images/COVID-655.png,COVID/masks/COVID-655.png,True,G03063,train,a928d3bc8c543883389bddbebed0fe61,c03c3f032d4d5e5b
COVID-656.png,COVID,COVID/images/COVID-656.png,COVID/masks/COVID-656.png,True,G03064,train,26cc5b8c730714008852f24212b5c6bc,8e0f40d41f4bcbcb
COVID-657.png,COVID,COVID/images/COVID-657.png,COVID/masks/COVID-657.png,True,G03065,train,c1bc8261badfd651b233633ecbf80c99,817f861a1d5f194d
COVID-658.png,COVID,COVID/images/COVID-658.png,COVID/masks/COVID-658.png,True,G03066,val,6af4a08e24188d78a7d53d2718e6e066,d47f23822b0dc357
COVID-659.png,COVID,COVID/images/COVID-659.png,COVID/masks/COVID-659.png,True,G03067,val,6c155e0ca5918896da11f057cc081b24,9b3bf0d406cad496
COVID-66.png,COVID,COVID/images/COVID-66.png,COVID/masks/COVID-66.png,True,G02623,train,355dcb0e8143cf63c94de9506f181234,d52ec2ff114d4d18
COVID-660.png,COVID,COVID/images/COVID-660.png,COVID/masks/COVID-660.png,True,G03068,val,dbbc5213477ce2935642cf338139a032,dd6815d7aa459692
COVID-661.png,COVID,COVID/images/COVID-661.png,COVID/masks/COVID-661.png,True,G00443,train,3681ae90940473e019fb52974034b99e,9c7dc0d207c71e2d
COVID-662.png,COVID,COVID/images/COVID-662.png,COVID/masks/COVID-662.png,True,G03069,test,616d503cec551c2e42861211c59d979d,c42eda8b2ca8e3e3
COVID-663.png,COVID,COVID/images/COVID-663.png,COVID/masks/COVID-663.png,True,G03070,test,6cd80186cd2c4b62ae73127dd9808706,987bc7161ec60f38
COVID-664.png,COVID,COVID/images/COVID-664.png,COVID/masks/COVID-664.png,True,G03071,train,6a9dfa838907818fa5b659bbda748795,d879c5c65e5e5a10
COVID-665.png,COVID,COVID/images/COVID-665.png,COVID/masks/COVID-665.png,True,G03072,train,cab16080411261e1e2448e84649b1b64,d42f23693cf261d4
COVID-666.png,COVID,COVID/images/COVID-666.png,COVID/masks/COVID-666.png,True,G03073,test,08328dfcbe5105c895477588833c2912,d939c51786469e4b
COVID-667.png,COVID,COVID/images/COVID-667.png,COVID/masks/COVID-667.png,True,G03074,train,bf8a506db45adff43be39a58bf8631fe,dd3a535f1c491d06
COVID-668.png,COVID,COVID/images/COVID-668.png,COVID/masks/COVID-668.png,True,G03075,train,cf894f486d6bc520c31fb12996f62818,887f961a1a596967
COVID-669.png,COVID,COVID/images/COVID-669.png,COVID/masks/COVID-669.png,True,G03076,train,7cd716b2eb92cc89b1e4696b08cded84,dd7dd01b4d0a1b64
COVID-67.png,COVID,COVID/images/COVID-67.png,COVID/masks/COVID-67.png,True,G03077,train,e7ff2831d0cb1fd208fcee2efe3b1f26,d42d9383df0b0f0e
COVID-670.png,COVID,COVID/images/COVID-670.png,COVID/masks/COVID-670.png,True,G03078,test,e53cb6c460334a111b017ce0da62b24f,987947565e58595a
COVID-671.png,COVID,COVID/images/COVID-671.png,COVID/masks/COVID-671.png,True,G03079,train,9406e7338ae60c381acbaa485021e5a6,817b101a19bbbfa6
COVID-672.png,COVID,COVID/images/COVID-672.png,COVID/masks/COVID-672.png,True,G03080,train,4bf7bf3d8366defe8ae868224ad86a65,d52e420b0f6c2e7e
COVID-673.png,COVID,COVID/images/COVID-673.png,COVID/masks/COVID-673.png,True,G03081,train,2b3c6771b953fb065a3bebbb6c8b485d,dd59521a5e1c1d0f
COVID-674.png,COVID,COVID/images/COVID-674.png,COVID/masks/COVID-674.png,True,G03082,train,af41e7c3b388d8bd9c0e0e47cc0ecfd8,d56a471a5e405f5c
COVID-675.png,COVID,COVID/images/COVID-675.png,COVID/masks/COVID-675.png,True,G03083,train,1d72f5135c058e94731d207e78287832,dc2f421d4c4c7cf2
COVID-676.png,COVID,COVID/images/COVID-676.png,COVID/masks/COVID-676.png,True,G03084,train,1d3a214a449d29098b6f4095231fb7c8,d919575e5e58580b
COVID-677.png,COVID,COVID/images/COVID-677.png,COVID/masks/COVID-677.png,True,G03085,train,578e6ec97ac888e43a49b914bee8f308,890f071e1a4d5f5e
COVID-678.png,COVID,COVID/images/COVID-678.png,COVID/masks/COVID-678.png,True,G03086,train,6bf6c29ce3beeec5394ea753db876f13,d12b571e19093da7
COVID-679.png,COVID,COVID/images/COVID-679.png,COVID/masks/COVID-679.png,True,G03087,train,867d1f3086cf4b87c0bdbc9514f61d42,d829c4161fc6395f
COVID-68.png,COVID,COVID/images/COVID-68.png,COVID/masks/COVID-68.png,True,G03088,train,4f147748b954c20949441d5ccec99c98,d42f434f4c6a6ce8
COVID-680.png,COVID,COVID/images/COVID-680.png,COVID/masks/COVID-680.png,True,G03089,train,92f473f2414240c0703011b05c47e79d,cc094b0c68aff5b5
COVID-681.png,COVID,COVID/images/COVID-681.png,COVID/masks/COVID-681.png,True,G03090,val,0972fd414cf68f7b9a4f3321a6bd3164,dc2b425f5d4c3c1c
COVID-682.png,COVID,COVID/images/COVID-682.png,COVID/masks/COVID-682.png,True,G03091,val,94dde04df265f96eb76d9c2a410d6b00,f86d851eb6d21ac4
COVID-683.png,COVID,COVID/images/COVID-683.png,COVID/masks/COVID-683.png,True,G03092,val,e31dd828fcf21fbae276e77e848fe103,cd45d21b4d4d5f41
COVID-684.png,COVID,COVID/images/COVID-684.png,COVID/masks/COVID-684.png,True,G02846,train,3946f9b3baf6943a091c146994b1cce3,dd2e40431b454f3f
COVID-685.png,COVID,COVID/images/COVID-685.png,COVID/masks/COVID-685.png,True,G03093,test,74e1a2b8bcea533330cca989f5960f3f,b979871e84171c4f
COVID-686.png,COVID,COVID/images/COVID-686.png,COVID/masks/COVID-686.png,True,G03094,test,0424b777da5c8817bfd950bccfc7ea43,917987473a16c756
COVID-687.png,COVID,COVID/images/COVID-687.png,COVID/masks/COVID-687.png,True,G00569,train,6b6dd927e2ecc588fecb42f1b51f4f47,b967960c961e869e
COVID-688.png,COVID,COVID/images/COVID-688.png,COVID/masks/COVID-688.png,True,G03095,train,92bd191de958737f99a071a4b97c1a99,9839c7871edec41c
COVID-689.png,COVID,COVID/images/COVID-689.png,COVID/masks/COVID-689.png,True,G03096,train,e325789e7dc4ab9ff86543a8e6da073e,d02f531e5d4b581e
COVID-69.png,COVID,COVID/images/COVID-69.png,COVID/masks/COVID-69.png,True,G03097,train,4a68ae08ceaabcd48b4fec84b677cb86,dc6de51616c3531a
COVID-690.png,COVID,COVID/images/COVID-690.png,COVID/masks/COVID-690.png,True,G03098,train,4ac96769601de257c98b68a862651970,9c7cc0d39b414d2f
COVID-691.png,COVID,COVID/images/COVID-691.png,COVID/masks/COVID-691.png,True,G03099,train,91a54f8623ebce7d36802ffe204a31fc,dc3e431f1f471c18
COVID-692.png,COVID,COVID/images/COVID-692.png,COVID/masks/COVID-692.png,True,G00450,train,faac30015eaf312cceb352cff3a3fe61,9c6cc30b4d5c5e4d
COVID-693.png,COVID,COVID/images/COVID-693.png,COVID/masks/COVID-693.png,True,G03100,train,9d6e02606b9249a30f77b798e7cb8b52,885d459796524b3f
COVID-694.png,COVID,COVID/images/COVID-694.png,COVID/masks/COVID-694.png,True,G03101,train,d846803e235deed27580d2e743667938,dc2e035d4c59e9f0
COVID-695.png,COVID,COVID/images/COVID-695.png,COVID/masks/COVID-695.png,True,G02908,train,9dcbad3dd9b99253b1ad40a48f20a585,95ee07d16b857885
COVID-696.png,COVID,COVID/images/COVID-696.png,COVID/masks/COVID-696.png,True,G03102,test,66bb5bab04c622aefd493522d63fc29f,cd5cd2d2cb49585c
COVID-697.png,COVID,COVID/images/COVID-697.png,COVID/masks/COVID-697.png,True,G03103,train,478386e239c567c8fe6aee8996c26007,dc2ed2434b574a1b
COVID-698.png,COVID,COVID/images/COVID-698.png,COVID/masks/COVID-698.png,True,G03104,train,3bdebc22b0f03512fb35bd29067eecb0,dd2ae7c0eaf4a0e0
COVID-699.png,COVID,COVID/images/COVID-699.png,COVID/masks/COVID-699.png,True,G03105,train,c99195a04dc376d94623d8b42be962c2,9c6dc2d21acb0d1f
COVID-7.png,COVID,COVID/images/COVID-7.png,COVID/masks/COVID-7.png,True,G03106,val,f200fee537ca6c1752de8a653fa93e48,887dc65e58d9c8e1
COVID-70.png,COVID,COVID/images/COVID-70.png,COVID/masks/COVID-70.png,True,G03107,train,3056a7668ac8b1ace1e1caf108eeac2f,d42fc2455b524f1d
COVID-700.png,COVID,COVID/images/COVID-700.png,COVID/masks/COVID-700.png,True,G03108,val,da04685f071ce0cb9d87b67c8f0b2a8d,817d06161e0f7f39
COVID-701.png,COVID,COVID/images/COVID-701.png,COVID/masks/COVID-701.png,True,G03109,test,20ae0fb28566962d15077906b7a9e656,c52ee2d189c5ac9e
COVID-702.png,COVID,COVID/images/COVID-702.png,COVID/masks/COVID-702.png,True,G03110,test,ce26feeb2e3b4253ce2252bdd4d8ae36,d94b1e2da39cc626
COVID-703.png,COVID,COVID/images/COVID-703.png,COVID/masks/COVID-703.png,True,G03111,train,61ef538dd94065138a1a8903a018c2f2,d92856d2c35f9e50
COVID-704.png,COVID,COVID/images/COVID-704.png,COVID/masks/COVID-704.png,True,G03112,train,e19438d63c2f3798ba2a8fdc24dfdf7a,817f461e1d495d1d
COVID-705.png,COVID,COVID/images/COVID-705.png,COVID/masks/COVID-705.png,True,G03113,train,75f77a7d1f80998a058cb2e671cb5004,9d79c71616d2c24e
COVID-706.png,COVID,COVID/images/COVID-706.png,COVID/masks/COVID-706.png,True,G03114,val,c3d884211f7dd4b6668c99c54496119a,817bc5065e1c5e5b
COVID-707.png,COVID,COVID/images/COVID-707.png,COVID/masks/COVID-707.png,True,G03115,train,18d33e7ac786bb455e83ca546592841b,dd2a864b1b555b1c
COVID-708.png,COVID,COVID/images/COVID-708.png,COVID/masks/COVID-708.png,True,G03116,train,1bb733ec50df4995194340b414f2ae78,c829e586bf409e5e
COVID-709.png,COVID,COVID/images/COVID-709.png,COVID/masks/COVID-709.png,True,G03117,train,0c3090821bf54606c5c97357c060ed50,dc2e434d0cb0bdb6
COVID-71.png,COVID,COVID/images/COVID-71.png,COVID/masks/COVID-71.png,True,G01385,test,15552e6d106c9498f451cf0e4f82a0ec,8178141e7b3b2b3b
COVID-710.png,COVID,COVID/images/COVID-710.png,COVID/masks/COVID-710.png,True,G03118,train,de62c37c7d1042fa6693be01b377fac2,8c2d60179717d717
COVID-711.png,COVID,COVID/images/COVID-711.png,COVID/masks/COVID-711.png,True,G03119,train,b45ddb81279079c0789d109565626e0e,d93d864b1e541b1d
COVID-712.png,COVID,COVID/images/COVID-712.png,COVID/masks/COVID-712.png,True,G03120,train,8c96d78821e9a975bfc2bf1582632f8e,9c6fe080a69f878b
COVID-713.png,COVID,COVID/images/COVID-713.png,COVID/masks/COVID-713.png,True,G03121,train,e8dfe8fa30570ef4c644ea2c156f4a44,893b864a1c5f5b1d
COVID-714.png,COVID,COVID/images/COVID-714.png,COVID/masks/COVID-714.png,True,G00510,train,8083f5b349490f2e6424ca1df7ff7bfe,996dc71a1a451b5d
COVID-715.png,COVID,COVID/images/COVID-715.png,COVID/masks/COVID-715.png,True,G03122,train,019459432c7936832c8ada0e04d829dc,9969c596ce1c1ec3
COVID-716.png,COVID,COVID/images/COVID-716.png,COVID/masks/COVID-716.png,True,G03123,train,3b2a1d2348ac9e366039d815f24bbdfc,ac2cd0930fcb974e
COVID-717.png,COVID,COVID/images/COVID-717.png,COVID/masks/COVID-717.png,True,G03124,train,eaff090964d2cb9505cfe4047dbf60ce,9d6ca492c29f6a93
COVID-718.png,COVID,COVID/images/COVID-718.png,COVID/masks/COVID-718.png,True,G03125,train,04c82dad263d6faa93454ee6db8845d2,d979c5961e58431e
COVID-719.png,COVID,COVID/images/COVID-719.png,COVID/masks/COVID-719.png,True,G03126,train,f574efac7518b9db2a061fbe6634be61,817f06ce0e1a5f0f
COVID-72.png,COVID,COVID/images/COVID-72.png,COVID/masks/COVID-72.png,True,G03127,train,052a5157059e28c2352c30310880ff44,d87be53616861e1a
COVID-720.png,COVID,COVID/images/COVID-720.png,COVID/masks/COVID-720.png,True,G03128,val,97522045bbfe11b54908526d9ae12027,9d6d64c4d61eda41
COVID-721.png,COVID,COVID/images/COVID-721.png,COVID/masks/COVID-721.png,True,G03129,train,13a1e7a0860ff7c61537f772553a4fa6,915b065e18594d7f
COVID-722.png,COVID,COVID/images/COVID-722.png,COVID/masks/COVID-722.png,True,G03130,val,de156cf8c9307f61cc96eaa8d258ccaa,d9fcc25ac45a1e68
COVID-723.png,COVID,COVID/images/COVID-723.png,COVID/masks/COVID-723.png,True,G03131,val,e4e6447d7b2de828bf9767017584b5f4,cc27470f4a696cea
COVID-724.png,COVID,COVID/images/COVID-724.png,COVID/masks/COVID-724.png,True,G00748,train,9304748fb29aa5a36ca75aad1d1a57be,9979c716015f185f
COVID-725.png,COVID,COVID/images/COVID-725.png,COVID/masks/COVID-725.png,True,G03132,test,8f1e158a20db3db8650295f1ddc21184,d42c43431fd3c34f
COVID-726.png,COVID,COVID/images/COVID-726.png,COVID/masks/COVID-726.png,True,G03133,test,ad919e5987fa4c58b1cb317d05d5c18f,d91186161aafa7a7
COVID-727.png,COVID,COVID/images/COVID-727.png,COVID/masks/COVID-727.png,True,G03134,test,c175335788c4e39d4ea4c44cf444c942,d93bd65e4a0a291d
COVID-728.png,COVID,COVID/images/COVID-728.png,COVID/masks/COVID-728.png,True,G03135,train,0447652245f0171534c6087d96d23779,887f87169a0d7f09
COVID-729.png,COVID,COVID/images/COVID-729.png,COVID/masks/COVID-729.png,True,G02951,test,0ce3258f81503d1524af62f62a08bda0,dc2a535f4a4c1c7c
COVID-73.png,COVID,COVID/images/COVID-73.png,COVID/masks/COVID-73.png,True,G03136,train,e558cd98cceddd72e5fb596ad650d5be,913846561f4f1e5b
COVID-730.png,COVID,COVID/images/COVID-730.png,COVID/masks/COVID-730.png,True,G03137,train,2f843c70041c6eed7e5dd90ff286a984,cc65c30fcb0d6d25
COVID-731.png,COVID,COVID/images/COVID-731.png,COVID/masks/COVID-731.png,True,G03138,train,7b775f3d46f6262f300e1ec5c3d254ad,9d2ec6131bc74f18
COVID-732.png,COVID,COVID/images/COVID-732.png,COVID/masks/COVID-732.png,True,G03139,train,f868c3378f5743c43b49cbcacd9f750e,9909c5470f5e1e9b
COVID-733.png,COVID,COVID/images/COVID-733.png,COVID/masks/COVID-733.png,True,G03140,train,48e7de46eb260a7dc8d97692908e2914,cc4f430f0f292d2f
COVID-734.png,COVID,COVID/images/COVID-734.png,COVID/masks/COVID-734.png,True,G03141,train,ff6e51a1f065c55c11e25ab9c05383bb,d86de4c6065f5a0b
COVID-735.png,COVID,COVID/images/COVID-735.png,COVID/masks/COVID-735.png,True,G03142,test,90d616c688773bcc31b4dcbaaec57da8,cc464b6de46de087
COVID-736.png,COVID,COVID/images/COVID-736.png,COVID/masks/COVID-736.png,True,G03143,train,5a0abd23c3d4036332237d64caceeb68,880d4547565e5f5b
COVID-737.png,COVID,COVID/images/COVID-737.png,COVID/masks/COVID-737.png,True,G03144,train,f4b349786f4674fa83611267811cd7aa,c82e524b1e1c3cbf
COVID-738.png,COVID,COVID/images/COVID-738.png,COVID/masks/COVID-738.png,True,G00362,train,9f217f4217a09cc588c1a4b717adfc7c,cc2dc252db531a5b
COVID-739.png,COVID,COVID/images/COVID-739.png,COVID/masks/COVID-739.png,True,G03145,train,813055c761b38044ae8aa4f3104e0a2a,817f1c1c1b397959
COVID-74.png,COVID,COVID/images/COVID-74.png,COVID/masks/COVID-74.png,True,G03146,train,bf94eb38e310d1b7a8e1ff7016f33d89,981fc61e1e1a1a3f
COVID-740.png,COVID,COVID/images/COVID-740.png,COVID/masks/COVID-740.png,True,G03147,train,bb6b8ebb8ef5b4f485f687eb6bb3a46a,817b06065e395d7b
COVID-741.png,COVID,COVID/images/COVID-741.png,COVID/masks/COVID-741.png,True,G03148,train,b928e3eac6e749979e7323705f2a46e0,85817a877ad2877a
COVID-742.png,COVID,COVID/images/COVID-742.png,COVID/masks/COVID-742.png,True,G03149,val,7415a2d820f6f2d365f4b28db6729805,d919071e1a5c4b5f
COVID-743.png,COVID,COVID/images/COVID-743.png,COVID/masks/COVID-743.png,True,G03150,train,c5d68e3b4bfe15fc0134d2d62ea5c529,817b0d541a597b5b
COVID-744.png,COVID,COVID/images/COVID-744.png,COVID/masks/COVID-744.png,True,G00944,train,bb8ad9e32b89eeb0ec9fb86e12837bb6,d42f23d12bc02f9d
COVID-745.png,COVID,COVID/images/COVID-745.png,COVID/masks/COVID-745.png,True,G03151,val,ec931b9dc1b99a9f6bbb9180207f3981,c96bd7525e545a18
COVID-746.png,COVID,COVID/images/COVID-746.png,COVID/masks/COVID-746.png,True,G03152,val,9f81cdd86961737f249f384f91db0568,cc2e464f0cece3b2
COVID-747.png,COVID,COVID/images/COVID-747.png,COVID/masks/COVID-747.png,True,G03153,test,998715ff2bbe838e65ac0f5e36524a82,d17bd7161c541a39
COVID-748.png,COVID,COVID/images/COVID-748.png,COVID/masks/COVID-748.png,True,G03154,test,804b1f041e3ba1c69a880df0ffdea55b,aa2bc5865eda4197
COVID-749.png,COVID,COVID/images/COVID-749.png,COVID/masks/COVID-749.png,True,G03155,train,ca081dd56c504a8ec2072b84d1a65255,997b471a164d1a5d
COVID-75.png,COVID,COVID/images/COVID-75.png,COVID/masks/COVID-75.png,True,G03156,train,d0922b6d31ffb84366c95de2f7a95126,987b65371e465a1a
COVID-750.png,COVID,COVID/images/COVID-750.png,COVID/masks/COVID-750.png,True,G03157,train,7304bf20b6e28465f3bf273c29c1b783,8c6dc3130f571b4b
COVID-751.png,COVID,COVID/images/COVID-751.png,COVID/masks/COVID-751.png,True,G03158,train,65a6920dd35ed90bd2a237b1c9950ac3,b979c6961a495f18
COVID-752.png,COVID,COVID/images/COVID-752.png,COVID/masks/COVID-752.png,True,G00321,test,522e423d2fc3be7d2303d5ab83625ad6,817b47161a4d5f1b
COVID-753.png,COVID,COVID/images/COVID-753.png,COVID/masks/COVID-753.png,True,G03159,train,24c85e44fb3fd0119ee243f4ce03ed7c,dc4d83161e171e57
COVID-754.png,COVID,COVID/images/COVID-754.png,COVID/masks/COVID-754.png,True,G03160,train,b0161ddd4f9d61f367920d7a0b2ba75b,897b861e195f3a38
COVID-755.png,COVID,COVID/images/COVID-755.png,COVID/masks/COVID-755.png,True,G00369,train,626a33f5e5cf6ab23e0e8c84759a0ca7,983dc55657c3461e
COVID-756.png,COVID,COVID/images/COVID-756.png,COVID/masks/COVID-756.png,True,G03161,train,232ea70423a2013f8b0545c07a075384,d97d460e1a474f38
COVID-757.png,COVID,COVID/images/COVID-757.png,COVID/masks/COVID-757.png,True,G03162,train,4124232b4dde74a0412ebcf308f2c492,907f465a596de0b2
COVID-758.png,COVID,COVID/images/COVID-758.png,COVID/masks/COVID-758.png,True,G03163,train,5560fe2cd22da93ed37040cea8de7c75,ae2f70490b686e1f
COVID-759.png,COVID,COVID/images/COVID-759.png,COVID/masks/COVID-759.png,True,G03164,train,a6f6aad1a069c3bc8a0356e2ba74ebd7,987d47561a4b5b1c
COVID-76.png,COVID,COVID/images/COVID-76.png,COVID/masks/COVID-76.png,True,G03165,train,20d628d662ed7344b26f59d2397d25e8,9c2c43596c6e5c6e
COVID-760.png,COVID,COVID/images/COVID-760.png,COVID/masks/COVID-760.png,True,G03166,train,f118d154d0904acb921daac32ae4cdde,946e035f085f4e5d
COVID-761.png,COVID,COVID/images/COVID-761.png,COVID/masks/COVID-761.png,True,G03167,train,2285ee1e3f6c70b81ef91c5faf43dbcf,9c2ec4d21a87e3da
COVID-762.png,COVID,COVID/images/COVID-762.png,COVID/masks/COVID-762.png,True,G03168,val,15c5683b886dab5dcd6326f4ccc9468e,995d43521acb1b9e
COVID-763.png,COVID,COVID/images/COVID-763.png,COVID/masks/COVID-763.png,True,G03169,test,aff5e047e3b5149ff0b7b3b976142a0c,817b161e191b5f39
COVID-764.png,COVID,COVID/images/COVID-764.png,COVID/masks/COVID-764.png,True,G00804,test,d5527256be674ac58f8ee589616b5989,817a161e597939e9
COVID-765.png,COVID,COVID/images/COVID-765.png,COVID/masks/COVID-765.png,True,G03170,train,cd285754d040adbc724d6820f832be2c,d42a93d32f9a3ad0
COVID-766.png,COVID,COVID/images/COVID-766.png,COVID/masks/COVID-766.png,True,G00157,val,5593eb8505e2734eb1e591aa9526a42c,d42aa70a5d5d193d
COVID-767.png,COVID,COVID/images/COVID-767.png,COVID/masks/COVID-767.png,True,G03171,val,dd4c6c840efc8a8e1fa3cbd467f27849,d42d0b2f2e3434b7
COVID-768.png,COVID,COVID/images/COVID-768.png,COVID/masks/COVID-768.png,True,G03172,test,858437d62968f4eed5f6219d77aba7e3,887c86da1bd35a1d
COVID-769.png,COVID,COVID/images/COVID-769.png,COVID/masks/COVID-769.png,True,G03173,train,920eff06b23fe91a76f1a0e8a0d7b111,d42e5141ad57691f
COVID-77.png,COVID,COVID/images/COVID-77.png,COVID/masks/COVID-77.png,True,G03034,test,df44588d06fbf4afd125bc1d12d7ebb5,9c36a20bc7170fc7
COVID-770.png,COVID,COVID/images/COVID-770.png,COVID/masks/COVID-770.png,True,G03133,test,8ee9552c0a1e99b921e72d01a1116d12,d91186161aafa7a7
COVID-771.png,COVID,COVID/images/COVID-771.png,COVID/masks/COVID-771.png,True,G03174,val,3c3e0d651ae97501c2857ef20d5526dd,9969c7161a5e1ec6
COVID-772.png,COVID,COVID/images/COVID-772.png,COVID/masks/COVID-772.png,True,G03175,train,347f1248c6628578ac63a5862058e50a,d528c75b554e1f48
COVID-773.png,COVID,COVID/images/COVID-773.png,COVID/masks/COVID-773.png,True,G00359,val,d829bdc1d0656118639afa3f6aa1c165,9c25c31ec35b1e69
COVID-774.png,COVID,COVID/images/COVID-774.png,COVID/masks/COVID-774.png,True,G03176,test,a2bce1063a3804da73370c4b96e4ad24,893bc71e96479658
COVID-775.png,COVID,COVID/images/COVID-775.png,COVID/masks/COVID-775.png,True,G03177,train,20b84e16ff79c6e4c430e60fdabef373,803c6f43ba13dbc3
COVID-776.png,COVID,COVID/images/COVID-776.png,COVID/masks/COVID-776.png,True,G03178,train,ef5537fd1c8601ebb4e7516333b269bc,cd2df01283471f1f
COVID-777.png,COVID,COVID/images/COVID-777.png,COVID/masks/COVID-777.png,True,G03179,train,d51464e248f846e812716bb8516bb182,917e16185b7b391a
COVID-778.png,COVID,COVID/images/COVID-778.png,COVID/masks/COVID-778.png,True,G03180,train,741ec9b0c3cb9ea345387c1684274097,9979c51707431a5f
COVID-779.png,COVID,COVID/images/COVID-779.png,COVID/masks/COVID-779.png,True,G03181,train,c663fe71945924cf515948f966efcd9a,d43ec3d31bc12f82
COVID-78.png,COVID,COVID/images/COVID-78.png,COVID/masks/COVID-78.png,True,G03182,train,92c1504df975aed3dbea13be638a28fb,9879e58616c7471e
COVID-780.png,COVID,COVID/images/COVID-780.png,COVID/masks/COVID-780.png,True,G03183,train,fc35d8a4ed2298e6a80edd21f6bfd9c6,9969c71643411f5f
COVID-781.png,COVID,COVID/images/COVID-781.png,COVID/masks/COVID-781.png,True,G03184,train,bfeec5b6e01dbac44abd2d56bb1e2151,c1697ed23a907ac3
COVID-782.png,COVID,COVID/images/COVID-782.png,COVID/masks/COVID-782.png,True,G00398,train,978f7454dee3c85f27997b250f02025d,d5ae861a1d1e1c5d
COVID-783.png,COVID,COVID/images/COVID-783.png,COVID/masks/COVID-783.png,True,G03185,val,4541dc41999bb3a68dca7e8fb12815c2,d47d430f0a2d2e3d
COVID-784.png,COVID,COVID/images/COVID-784.png,COVID/masks/COVID-784.png,True,G03186,train,18903ae5aff2c0465736af6ae301c293,d129c712be6c3b38
COVID-785.png,COVID,COVID/images/COVID-785.png,COVID/masks/COVID-785.png,True,G03187,train,ca53e5635812861494901a8396c27d60,d43d535f0a2c2f2c
COVID-786.png,COVID,COVID/images/COVID-786.png,COVID/masks/COVID-786.png,True,G00449,test,86e3e601eff60476cc8a9bc5b062a7f5,817d87963a292d3d
COVID-787.png,COVID,COVID/images/COVID-787.png,COVID/masks/COVID-787.png,True,G03188,train,771916cce6b5efa715cef1169f516097,897f161a185b5b78
COVID-788.png,COVID,COVID/images/COVID-788.png,COVID/masks/COVID-788.png,True,G03189,val,601217d3c84bc621876920e367547fea,de2c495b4a2e2d2e
COVID-789.png,COVID,COVID/images/COVID-789.png,COVID/masks/COVID-789.png,True,G00253,train,f142411a943cc1c96233a87c1ac3379c,d92f861e04175e5e
COVID-79.png,COVID,COVID/images/COVID-79.png,COVID/masks/COVID-79.png,True,G03190,train,d50213a7c5d24f6a6b04b1a04eec19ee,d52a47435b5b4353
COVID-790.png,COVID,COVID/images/COVID-790.png,COVID/masks/COVID-790.png,True,G03191,val,553aa74f598f3dd5d671abdb785b623b,d56e06161b4b1b5d
COVID-791.png,COVID,COVID/images/COVID-791.png,COVID/masks/COVID-791.png,True,G03192,test,470dcce4ba50743e872224352bd71de6,c1a93ed60e8a9e1e
COVID-792.png,COVID,COVID/images/COVID-792.png,COVID/masks/COVID-792.png,True,G00098,train,346136eb90fabdf02c5629fbd5d33419,f969c6165e461a6a
COVID-793.png,COVID,COVID/images/COVID-793.png,COVID/masks/COVID-793.png,True,G03193,test,d518e013f1c49741cb40589322bf62eb,d42d03575e0e4b7c
COVID-794.png,COVID,COVID/images/COVID-794.png,COVID/masks/COVID-794.png,True,G00388,train,e0270fb143857b3a26ae4cd78f160117,d829b59687965a1e
COVID-795.png,COVID,COVID/images/COVID-795.png,COVID/masks/COVID-795.png,True,G03194,test,9261d024f4d20471d46b1acd7a09eb20,fd2c474b5d4d4c0a
COVID-796.png,COVID,COVID/images/COVID-796.png,COVID/masks/COVID-796.png,True,G03195,train,647893d301d926c5ce5a94a92d2422f9,953d061a38cd7bd8
COVID-797.png,COVID,COVID/images/COVID-797.png,COVID/masks/COVID-797.png,True,G03196,train,15e4b6bcfa1d4ffee99981aef3617b3b,dd45820b0f0e6d6f
COVID-798.png,COVID,COVID/images/COVID-798.png,COVID/masks/COVID-798.png,True,G03197,train,8621e539b53d56eda32fdccef4541eba,897d065879e2f2e2
COVID-799.png,COVID,COVID/images/COVID-799.png,COVID/masks/COVID-799.png,True,G03198,train,1dbf3087e1a59a97e7da61e72d169eb6,dd2c421b55473d69
COVID-8.png,COVID,COVID/images/COVID-8.png,COVID/masks/COVID-8.png,True,G00627,train,ebdc3d131086d8927f50b56edacc864a,d83ac3879ec6523a
COVID-80.png,COVID,COVID/images/COVID-80.png,COVID/masks/COVID-80.png,True,G03172,test,414e72b5d4975554150670939cf3fe56,887c86da1bd35a1d
COVID-800.png,COVID,COVID/images/COVID-800.png,COVID/masks/COVID-800.png,True,G03199,train,ab582b48c01fe6ea5bbf9c29bfd63809,dd28c6430f551b1f
COVID-801.png,COVID,COVID/images/COVID-801.png,COVID/masks/COVID-801.png,True,G03200,val,6c725629c8bcce31ac347f2446ae46f0,d42ed14b1d556a1d
COVID-802.png,COVID,COVID/images/COVID-802.png,COVID/masks/COVID-802.png,True,G03201,train,49583ea7002ae691b55b0e9b00f8698e,997f06184d78b9e8
COVID-803.png,COVID,COVID/images/COVID-803.png,COVID/masks/COVID-803.png,True,G03202,train,deebb2f78f870e59342d0999f389b73b,dc2ed3430b432b2f
COVID-804.png,COVID,COVID/images/COVID-804.png,COVID/masks/COVID-804.png,True,G02962,train,92119cd41b554a089ff5e9f57b29bd34,cd4d521a1b4d2f47
COVID-805.png,COVID,COVID/images/COVID-805.png,COVID/masks/COVID-805.png,True,G03203,train,6837b7b9ef8f915911f2ca6625d4e717,dd2ed30f47060e4d
COVID-806.png,COVID,COVID/images/COVID-806.png,COVID/masks/COVID-806.png,True,G03204,train,65c313fc07c8354f5ecb7f7861dbae14,d55fa2a31f05415d
COVID-807.png,COVID,COVID/images/COVID-807.png,COVID/masks/COVID-807.png,True,G03205,train,1036946fb8ef833b38b461b0f135a963,d43d50e7932bd50a
COVID-808.png,COVID,COVID/images/COVID-808.png,COVID/masks/COVID-808.png,True,G03206,train,2d83287571f08f6fc26b94ab46a0d1a9,957de2c20b4f494d
COVID-809.png,COVID,COVID/images/COVID-809.png,COVID/masks/COVID-809.png,True,G03207,train,e7f2ea60070b546e19160479e2653514,d47d62d70a595f40
COVID-81.png,COVID,COVID/images/COVID-81.png,COVID/masks/COVID-81.png,True,G03208,train,0e86fe243e52794e93d6494b9a98ec65,997be5053ed402bc
COVID-810.png,COVID,COVID/images/COVID-810.png,COVID/masks/COVID-810.png,True,G03209,train,6aee0e3c78c51ff37cbd03ac7e46558d,d0fdc34b4e0b0d4b
COVID-811.png,COVID,COVID/images/COVID-811.png,COVID/masks/COVID-811.png,True,G03210,val,3f8943ddc8626c26a3e63b26aefc8bee,cc6c831b1b1b1b4f
COVID-812.png,COVID,COVID/images/COVID-812.png,COVID/masks/COVID-812.png,True,G03211,val,70987d0764a16feb72ebe7de2bb2d254,807f87165d4d4f1c
COVID-813.png,COVID,COVID/images/COVID-813.png,COVID/masks/COVID-813.png,True,G03212,test,b8e17d8b30df460130634937ef812a48,dd2952d7474a5552
COVID-814.png,COVID,COVID/images/COVID-814.png,COVID/masks/COVID-814.png,True,G03213,test,c53ea8ca2714540fb83151b6eb7f9b6b,897d061e1b595d78
COVID-815.png,COVID,COVID/images/COVID-815.png,COVID/masks/COVID-815.png,True,G03214,train,f0d960343d03f55ccb0946d7feae7fd1,8c4c861f4d4e5e4f
COVID-816.png,COVID,COVID/images/COVID-816.png,COVID/masks/COVID-816.png,True,G03215,test,a965d789d81876c2343760bf63c487bd,886dc75e5d5a1949
COVID-817.png,COVID,COVID/images/COVID-817.png,COVID/masks/COVID-817.png,True,G03216,train,12061f9da4070da6fb99ab7542f659e3,dc28534bc31fc90f
COVID-818.png,COVID,COVID/images/COVID-818.png,COVID/masks/COVID-818.png,True,G03217,train,99c9a502989b7954f68069a78ad09833,dd2cd24b4f0f4c4c
COVID-819.png,COVID,COVID/images/COVID-819.png,COVID/masks/COVID-819.png,True,G03218,train,4ae336065e5c6824351d400d662f6cfd,986d071e455f5a4d
COVID-82.png,COVID,COVID/images/COVID-82.png,COVID/masks/COVID-82.png,True,G03219,train,96f430aba09470ae50a82e61cbf75186,dd2a83431d0d0f5f
COVID-820.png,COVID,COVID/images/COVID-820.png,COVID/masks/COVID-820.png,True,G00394,train,2ef30a5f396b8fb9c06fa4936db7bf52,c83975021736567f
COVID-821.png,COVID,COVID/images/COVID-821.png,COVID/masks/COVID-821.png,True,G03220,train,be64883e67df794c7c0b9b32502f62c6,9879c71e1f571a50
COVID-822.png,COVID,COVID/images/COVID-822.png,COVID/masks/COVID-822.png,True,G03221,train,f4ea315594e0fa4e5e3f34ff01ea4a76,886f831b2d2d2f39
COVID-823.png,COVID,COVID/images/COVID-823.png,COVID/masks/COVID-823.png,True,G03222,train,da4cf2f0ef5902c9253bdf18b6191614,d92bc21f4f021f5a
COVID-824.png,COVID,COVID/images/COVID-824.png,COVID/masks/COVID-824.png,True,G03223,train,4e8ada71d836a9b0205d41eff7b4c4bf,91784c5e68e3a5e5
COVID-825.png,COVID,COVID/images/COVID-825.png,COVID/masks/COVID-825.png,True,G03224,val,3f80a4d13e0bdc4c5e967c4f697ce041,894d06c30fcb0fcf
COVID-826.png,COVID,COVID/images/COVID-826.png,COVID/masks/COVID-826.png,True,G03225,train,9f8ead02227924aa64308b983f7100f5,c625ca07ab27b5b4
COVID-827.png,COVID,COVID/images/COVID-827.png,COVID/masks/COVID-827.png,True,G03226,train,3b853ddc8b74f35f15930f0d0cf8fbd0,d52a6f465e0d194d
COVID-828.png,COVID,COVID/images/COVID-828.png,COVID/masks/COVID-828.png,True,G03227,train,5628675d55df942df0016d21f32e324d,992dc7565a59194d
COVID-829.png,COVID,COVID/images/COVID-829.png,COVID/masks/COVID-829.png,True,G03228,train,e2a323b532178af0611b5d6d343744ba,dd64530fc61ec60e
COVID-83.png,COVID,COVID/images/COVID-83.png,COVID/masks/COVID-83.png,True,G03163,train,429fe0f9c7d0c9d7b287707fcad3941d,ae2f70490b686e1f
COVID-830.png,COVID,COVID/images/COVID-830.png,COVID/masks/COVID-830.png,True,G03229,train,43a37810b5fe9b3304f4fa5b502602d3,89584c1ffaa3e5a4
COVID-831.png,COVID,COVID/images/COVID-831.png,COVID/masks/COVID-831.png,True,G03230,val,ba896e4d4a6404da5b74fe925346e815,897b0796dc591596
COVID-832.png,COVID,COVID/images/COVID-832.png,COVID/masks/COVID-832.png,True,G03231,train,9128b54fba373e88cb6678f771be953e,d933d61e5e495950
COVID-833.png,COVID,COVID/images/COVID-833.png,COVID/masks/COVID-833.png,True,G03232,val,5f047ea1cc31c71e327d34da67d5cab5,817d46861a195f5f
COVID-834.png,COVID,COVID/images/COVID-834.png,COVID/masks/COVID-834.png,True,G03233,test,33518344f3f6aa71f17bf5d476721276,985b47c65e5a491d
COVID-835.png,COVID,COVID/images/COVID-835.png,COVID/masks/COVID-835.png,True,G03234,train,124b3d1bb2eefb2ea643c1eade985a3e,dd2e035f5d0f1845
COVID-836.png,COVID,COVID/images/COVID-836.png,COVID/masks/COVID-836.png,True,G03235,test,0266c70737db05a8c725f5b95581266d,817f061e5b7868f8
COVID-837.png,COVID,COVID/images/COVID-837.png,COVID/masks/COVID-837.png,True,G03236,test,d1badb7dcec34bfc3f092c51a2a7d72b,d42e0b0b0b2e6b3f
COVID-838.png,COVID,COVID/images/COVID-838.png,COVID/masks/COVID-838.png,True,G03237,train,99a08f38ffbe3883f28b9dfa5b9276da,cd2d82538b1f1f0d
COVID-839.png,COVID,COVID/images/COVID-839.png,COVID/masks/COVID-839.png,True,G03238,train,2f38650dbefea09525843d7165770b73,d47ac35b198c0f39
COVID-84.png,COVID,COVID/images/COVID-84.png,COVID/masks/COVID-84.png,True,G03239,train,86492f84a872b7d8d28f868c5f4bbbcf,9d6da192120f1f1f
COVID-840.png,COVID,COVID/images/COVID-840.png,COVID/masks/COVID-840.png,True,G03240,train,a497ceaeed065c93ed3b62568bdf3c84,d42f534b0f4a2d2e
COVID-841.png,COVID,COVID/images/COVID-841.png,COVID/masks/COVID-841.png,True,G03241,train,b58916f1bf0813fdf1f33eff76de1db3,d42e430b0cec3cfd
COVID-842.png,COVID,COVID/images/COVID-842.png,COVID/masks/COVID-842.png,True,G03242,train,91feb136e43c4b5c411683968cd7e67a,d81b522c2d6565e7
COVID-843.png,COVID,COVID/images/COVID-843.png,COVID/masks/COVID-843.png,True,G03243,train,ce57bec8aee32abb7faaa0e60164ffe7,d42853c35f5d4a0f
COVID-844.png,COVID,COVID/images/COVID-844.png,COVID/masks/COVID-844.png,True,G03244,train,481013b6e3809cf182a867d3450b7f77,d46a410b2f3e3c76
COVID-845.png,COVID,COVID/images/COVID-845.png,COVID/masks/COVID-845.png,True,G03245,train,8d6f0df8cec9e0f537ec07ba759a0578,887e465a696372e3
COVID-846.png,COVID,COVID/images/COVID-846.png,COVID/masks/COVID-846.png,True,G00478,train,882b7941f347ccabdf1721cc28679198,d939c7961e4a3958
COVID-847.png,COVID,COVID/images/COVID-847.png,COVID/masks/COVID-847.png,True,G03246,train,6b5ad7caa73bd30fd34e62e62a8e9444,9959961a5bdbb8a0
COVID-848.png,COVID,COVID/images/COVID-848.png,COVID/masks/COVID-848.png,True,G01938,val,3271b8363abde82d1945b543b35dca4e,d83166c24eca1fdc
COVID-849.png,COVID,COVID/images/COVID-849.png,COVID/masks/COVID-849.png,True,G03247,train,626b3c7edee9177a07937e901c5d0bec,956c87165b073c5e
COVID-85.png,COVID,COVID/images/COVID-85.png,COVID/masks/COVID-85.png,True,G03248,train,21c7f4aaca515ba338aa42a8d96d908c,d13a47435a5f4a4b
COVID-850.png,COVID,COVID/images/COVID-850.png,COVID/masks/COVID-850.png,True,G03249,val,709edf62f39aacc3718f0bbc9cb883ca,d42f414f0f2a3cf1
COVID-851.png,COVID,COVID/images/COVID-851.png,COVID/masks/COVID-851.png,True,G03250,val,cd4dc0bcd60099e74145698e7584cfbe,d42e831f1d5f2e18
COVID-852.png,COVID,COVID/images/COVID-852.png,COVID/masks/COVID-852.png,True,G03251,val,249e3e3399cba2340237abede5666aee,917847471a1b4d6f
COVID-853.png,COVID,COVID/images/COVID-853.png,COVID/masks/COVID-853.png,True,G03252,test,69f6d9aa0cbe5a8854f179dde3ad7e2f,817b561a5c4f1e39
COVID-854.png,COVID,COVID/images/COVID-854.png,COVID/masks/COVID-854.png,True,G03253,test,c2ea5fddf09475d3ee0dd8ecbe8746b9,dd2a531f580b3969
COVID-855.png,COVID,COVID/images/COVID-855.png,COVID/masks/COVID-855.png,True,G03254,train,f76603e4682d6e60a64331b45b52cdf4,917a57475a1b0b0f
COVID-856.png,COVID,COVID/images/COVID-856.png,COVID/masks/COVID-856.png,True,G03255,test,dbfd32804d5d6ec5ebd13a02280e5fbe,817b96861b1d7a59
COVID-857.png,COVID,COVID/images/COVID-857.png,COVID/masks/COVID-857.png,True,G03256,train,29d292c2fa2d91330af04ed8ee92d893,c17b76561050de5e
COVID-858.png,COVID,COVID/images/COVID-858.png,COVID/masks/COVID-858.png,True,G03257,train,02dfd8b89c3e3d8a47c0956555497027,d42f434f2c6c6ce4
COVID-859.png,COVID,COVID/images/COVID-859.png,COVID/masks/COVID-859.png,True,G03258,train,afc7296f236a3b6ad69bc17366e1b1e7,903a121afab6b5e5
COVID-86.png,COVID,COVID/images/COVID-86.png,COVID/masks/COVID-86.png,True,G00748,train,4e5903401ae06bfc7436fb773e1b8309,9979c716015f185f
COVID-860.png,COVID,COVID/images/COVID-860.png,COVID/masks/COVID-860.png,True,G03259,train,f2e4ae414f6c80b0be3a1e5c1e774000,d42f535b0b0e1b4d
COVID-861.png,COVID,COVID/images/COVID-861.png,COVID/masks/COVID-861.png,True,G03260,train,dcdc2c31aa72773995c7cbd7ef25ec34,817e0e1e5b4d7878
COVID-862.png,COVID,COVID/images/COVID-862.png,COVID/masks/COVID-862.png,True,G03261,train,f62fa7ab64bf3a5a3816766562b5263d,887d0787121dff0d
COVID-863.png,COVID,COVID/images/COVID-863.png,COVID/masks/COVID-863.png,True,G03262,train,2c3ca78cf0c220cf0dc02d8e981bd699,d92f861e151acd5a
COVID-864.png,COVID,COVID/images/COVID-864.png,COVID/masks/COVID-864.png,True,G03263,train,272124a87ccd9880240d475894fe65bf,917d061a7a9a7ab4
COVID-865.png,COVID,COVID/images/COVID-865.png,COVID/masks/COVID-865.png,True,G03264,train,e3cb2f8d8784911f64343e4be2157d98,dcadd24fd44a1d44
COVID-866.png,COVID,COVID/images/COVID-866.png,COVID/masks/COVID-866.png,True,G03265,train,fbe983eba707843734bbab0c09c0471b,d13b561e1839e9e1
COVID-867.png,COVID,COVID/images/COVID-867.png,COVID/masks/COVID-867.png,True,G03266,train,e06ee00352da6aad9e3b5ee1e277ed8b,887f021a4f5a1f6e
COVID-868.png,COVID,COVID/images/COVID-868.png,COVID/masks/COVID-868.png,True,G03267,train,21de73578fbe01aefa965877a569ae22,887d06565a595d5d
COVID-869.png,COVID,COVID/images/COVID-869.png,COVID/masks/COVID-869.png,True,G03268,val,00370d6ce9b836de5f20e931802640a6,997f06161a4f5e38
COVID-87.png,COVID,COVID/images/COVID-87.png,COVID/masks/COVID-87.png,True,G03269,val,8761d1e481cf10a7cbf731676f380b78,d839c546964517be
COVID-870.png,COVID,COVID/images/COVID-870.png,COVID/masks/COVID-870.png,True,G03270,test,c24a4f43d0f6c1d0087d35ac8f4782fc,817b071e1e0f5f1a
COVID-871.png,COVID,COVID/images/COVID-871.png,COVID/masks/COVID-871.png,True,G03271,train,41b000429d0f8232e8ba1f2f48637523,c54f034d6ce2d2f1
COVID-872.png,COVID,COVID/images/COVID-872.png,COVID/masks/COVID-872.png,True,G03272,test,e3a77cb318343c7d86a1ce5d03b8d174,995a165a5b79a0e5
COVID-873.png,COVID,COVID/images/COVID-873.png,COVID/masks/COVID-873.png,True,G00401,test,b885ab6937cd9d6a59e7bf3ba2fe5bc3,d93de7474e0a1839
COVID-874.png,COVID,COVID/images/COVID-874.png,COVID/masks/COVID-874.png,True,G03273,val,6dc27255ff977489cd8b485fe1bde700,996d861a071f1b5e
COVID-875.png,COVID,COVID/images/COVID-875.png,COVID/masks/COVID-875.png,True,G03274,train,d7ed06641e9746d4732f61670e0a07e8,805f475e182d2dbe
COVID-876.png,COVID,COVID/images/COVID-876.png,COVID/masks/COVID-876.png,True,G03275,train,01fb32ff4dfbc443cdfc5842cb95a658,cd2e93c30fc40ed5
COVID-877.png,COVID,COVID/images/COVID-877.png,COVID/masks/COVID-877.png,True,G03276,test,e4d494567f93d0e7627dd09c0c8cda13,c94a560a1d2d3dbd
COVID-878.png,COVID,COVID/images/COVID-878.png,COVID/masks/COVID-878.png,True,G03277,train,acc5510d7c5c404ef28d965a7e38f66a,917bd61e58511a5b
COVID-879.png,COVID,COVID/images/COVID-879.png,COVID/masks/COVID-879.png,True,G03278,train,e5171072e6286ac73c5d54e9c226c313,d52e22534b4d4d4f
COVID-88.png,COVID,COVID/images/COVID-88.png,COVID/masks/COVID-88.png,True,G03279,train,d2a0247e2981c1dae0c70c782740deb9,907fc21e1e084f7d
COVID-880.png,COVID,COVID/images/COVID-880.png,COVID/masks/COVID-880.png,True,G03280,train,f7211b9e6740ce142068c41fac473c40,f96ad6da1082a6ab
COVID-881.png,COVID,COVID/images/COVID-881.png,COVID/masks/COVID-881.png,True,G03281,train,7e8e62ff4697bfa64813501a80c36355,dd2a534b0f3c281f
COVID-882.png,COVID,COVID/images/COVID-882.png,COVID/masks/COVID-882.png,True,G03282,train,2330696f70323991ca9711be3405ce08,8d6a571a7d617522
COVID-883.png,COVID,COVID/images/COVID-883.png,COVID/masks/COVID-883.png,True,G03283,train,f9d09253e819afa86fdb607c9f3f57ab,f95947161a594d1e
COVID-884.png,COVID,COVID/images/COVID-884.png,COVID/masks/COVID-884.png,True,G03284,train,cbd72d3aa2ad26a40508950d71009f8b,8c0d03171e1f3f3b
COVID-885.png,COVID,COVID/images/COVID-885.png,COVID/masks/COVID-885.png,True,G03285,train,c54060e176174fb65837e68f955aa708,946e83131b151f3f
COVID-886.png,COVID,COVID/images/COVID-886.png,COVID/masks/COVID-886.png,True,G03286,train,5285515cf7cc076a989e54159c3bd2e7,dd68d21f4fd01869
COVID-887.png,COVID,COVID/images/COVID-887.png,COVID/masks/COVID-887.png,True,G03287,train,ac5de1d4777d04a0252cf2dec040b448,ae21c10e4a4fe97d
COVID-888.png,COVID,COVID/images/COVID-888.png,COVID/masks/COVID-888.png,True,G03288,train,83ffbb003bb4f682c9f9b659e5c28968,dd2ac64b0f540b6d
COVID-889.png,COVID,COVID/images/COVID-889.png,COVID/masks/COVID-889.png,True,G03289,train,39cd9ff0b92063bd0830529e8f0188a9,c1e02f873e96321f
COVID-89.png,COVID,COVID/images/COVID-89.png,COVID/masks/COVID-89.png,True,G03290,val,af0f7c81fe7dcea67ed705d2fc6af1d9,917b46d61a4f1f18
COVID-890.png,COVID,COVID/images/COVID-890.png,COVID/masks/COVID-890.png,True,G03291,test,33625c83609e9808a09320182affc170,987f07161e055f5a
COVID-891.png,COVID,COVID/images/COVID-891.png,COVID/masks/COVID-891.png,True,G03292,val,9e038cb6a1bec7e53840e9858846dde3,d42e035d4c6c78fa
COVID-892.png,COVID,COVID/images/COVID-892.png,COVID/masks/COVID-892.png,True,G03293,val,afb245b48284179a4d6ee4c7e7590c91,b979c5971e58411d
COVID-893.png,COVID,COVID/images/COVID-893.png,COVID/masks/COVID-893.png,True,G03294,test,01d4ee83c4e37ef0a398356df691cca8,c54f024b0f4f4d6d
COVID-894.png,COVID,COVID/images/COVID-894.png,COVID/masks/COVID-894.png,True,G03295,train,c5c91fa53177cd5c653a010a77ff47a7,8179ce1618cd1f3b
COVID-895.png,COVID,COVID/images/COVID-895.png,COVID/masks/COVID-895.png,True,G03296,train,25076edcf07005b5b066a5d6acba7b6b,d17b1a1b1d3b1859
COVID-896.png,COVID,COVID/images/COVID-896.png,COVID/masks/COVID-896.png,True,G03297,train,a664a0ff16202cc41d006752c08ddf43,813d0c1c1b1f5f79
COVID-897.png,COVID,COVID/images/COVID-897.png,COVID/masks/COVID-897.png,True,G03298,test,e3a864a8532033100e735863d8178bee,817906961f8f8f1b
COVID-898.png,COVID,COVID/images/COVID-898.png,COVID/masks/COVID-898.png,True,G02954,test,6f1c345b61903fa3582632fc87f97365,986d075ec21f5a69
COVID-899.png,COVID,COVID/images/COVID-899.png,COVID/masks/COVID-899.png,True,G03299,train,1cc46615d8ec61fd68f5c1dcdc5d7615,cd1b4e16165e1b2d
COVID-9.png,COVID,COVID/images/COVID-9.png,COVID/masks/COVID-9.png,True,G03300,train,70009596389669d54262dc59f5f471ce,802bd61e19593f67
COVID-90.png,COVID,COVID/images/COVID-90.png,COVID/masks/COVID-90.png,True,G00142,train,52950c97b67eba7754867189523d82ae,987d8716174f1707
COVID-900.png,COVID,COVID/images/COVID-900.png,COVID/masks/COVID-900.png,True,G03301,train,50d64cbecef0606b90de70186151dba1,817f061a5f4e7869
COVID-901.png,COVID,COVID/images/COVID-901.png,COVID/masks/COVID-901.png,True,G03302,train,f37d6eb0b4be51afb4c4160715f10a2b,cc6c431b0c6cecf6
COVID-902.png,COVID,COVID/images/COVID-902.png,COVID/masks/COVID-902.png,True,G03303,val,65e9d9f3a510b9227a3be1b0791a492a,cc65c31b5b431e47
COVID-903.png,COVID,COVID/images/COVID-903.png,COVID/masks/COVID-903.png,True,G03304,val,5d24dc2bd837281a24ada31bd31e620a,ae25d1c50fc3c92e
COVID-904.png,COVID,COVID/images/COVID-904.png,COVID/masks/COVID-904.png,True,G03305,test,6dbfc8441716aec6f21d5de71b67e82a,dc0a0b1d6e72b4b5
COVID-905.png,COVID,COVID/images/COVID-905.png,COVID/masks/COVID-905.png,True,G03306,train,0f67d83e9da6a9944ee9aaf63b9918b8,dc6cc3da496b5948
COVID-906.png,COVID,COVID/images/COVID-906.png,COVID/masks/COVID-906.png,True,G03307,val,90b06c6e3d564681d0d08187d6abd60a,dc4c434b4f4d4dac
COVID-907.png,COVID,COVID/images/COVID-907.png,COVID/masks/COVID-907.png,True,G03308,train,26cc39c5dfd3032fc88c9c6c54f3e0f2,887d071a135f5f49
COVID-908.png,COVID,COVID/images/COVID-908.png,COVID/masks/COVID-908.png,True,G03309,train,a3c894c538e7f8067d102a7595919d7a,817d06161b597d79
COVID-909.png,COVID,COVID/images/COVID-909.png,COVID/masks/COVID-909.png,True,G03310,train,f352c1f89b3aa1ce0ec11291086009aa,d57e475f584d4502
COVID-91.png,COVID,COVID/images/COVID-91.png,COVID/masks/COVID-91.png,True,G03311,train,e6ab66ae18c915b52df0b43ec16c86d9,912dc616d547bb15
COVID-910.png,COVID,COVID/images/COVID-910.png,COVID/masks/COVID-910.png,True,G03312,train,64b54bfc3812a74bb9e174a5cfb6075e,d52c52d34b1b4e4b
COVID-911.png,COVID,COVID/images/COVID-911.png,COVID/masks/COVID-911.png,True,G03313,test,e853d85e3efcb0648f335bd0fe42f777,887b05d7165a5b59
COVID-912.png,COVID,COVID/images/COVID-912.png,COVID/masks/COVID-912.png,True,G03314,test,9c1ded414c099bc81c52fb32021d7497,dd28c7821f8b531d
COVID-913.png,COVID,COVID/images/COVID-913.png,COVID/masks/COVID-913.png,True,G03315,train,e20635b03dadc882171b341a9d08e3f9,817b165a596979b8
COVID-914.png,COVID,COVID/images/COVID-914.png,COVID/masks/COVID-914.png,True,G03316,train,1a288a0dff0f2e7821aa6253c3518950,d232143f3bafec80
COVID-915.png,COVID,COVID/images/COVID-915.png,COVID/masks/COVID-915.png,True,G03317,train,f879d35004ad117d3f4cc5e6df3069a7,d42ea3c357555545
COVID-916.png,COVID,COVID/images/COVID-916.png,COVID/masks/COVID-916.png,True,G03318,train,68be317d768d0fe978bd1b5c43c3b967,c44a17cf2fc82fc1
COVID-917.png,COVID,COVID/images/COVID-917.png,COVID/masks/COVID-917.png,True,G03319,train,6341e76ba1c5085aa661dcc96098146b,817904041e7f5f5f
COVID-918.png,COVID,COVID/images/COVID-918.png,COVID/masks/COVID-918.png,True,G03320,train,33f1026bf5b9add23e04ef0b209d216e,987f5c5879e5e0c0
COVID-919.png,COVID,COVID/images/COVID-919.png,COVID/masks/COVID-919.png,True,G03321,train,6b3cc79563562ef21debbfdeeceedd0d,d52e56570b0b0f4d
COVID-92.png,COVID,COVID/images/COVID-92.png,COVID/masks/COVID-92.png,True,G03322,train,2262deee3794dda44e6095ac9e53912a,917c461e394d7d58
COVID-920.png,COVID,COVID/images/COVID-920.png,COVID/masks/COVID-920.png,True,G03323,train,b169912dfc4ba968bc43b1d5ada6a908,d43f435b590c2f19
COVID-921.png,COVID,COVID/images/COVID-921.png,COVID/masks/COVID-921.png,True,G03324,train,66d776e786ebc77315346a2e166a3f7b,8c6cc2938bc98f1f
COVID-922.png,COVID,COVID/images/COVID-922.png,COVID/masks/COVID-922.png,True,G03325,train,a7ba6411011e88212f18cb7a90258e74,d52e871f0d4c2d0f
COVID-923.png,COVID,COVID/images/COVID-923.png,COVID/masks/COVID-923.png,True,G03326,train,728a2372b6c5eea501f22c6b6b75f411,817a5616787f3b28
COVID-924.png,COVID,COVID/images/COVID-924.png,COVID/masks/COVID-924.png,True,G03327,train,9c49ab391ab6ecc5c480bec48948e6bd,ce25134b0f0e2f2f
COVID-925.png,COVID,COVID/images/COVID-925.png,COVID/masks/COVID-925.png,True,G02828,train,2df56694b1851259eccb988dab3e9eb6,df2da15ee2127a0c
COVID-926.png,COVID,COVID/images/COVID-926.png,COVID/masks/COVID-926.png,True,G03328,train,a20d548f780240ce28b33285152c9505,dd29c7571a411e1e
COVID-927.png,COVID,COVID/images/COVID-927.png,COVID/masks/COVID-927.png,True,G03329,val,60b6eb9da5a7a38663d499213738e6a5,a93fd60c9e883d1c
COVID-928.png,COVID,COVID/images/COVID-928.png,COVID/masks/COVID-928.png,True,G03330,val,6a89cd446af17f89f8f59766fd57b7e6,d52f434b0e6e6e42
COVID-929.png,COVID,COVID/images/COVID-929.png,COVID/masks/COVID-929.png,True,G03331,val,c4bbcc88cf8a13e003d420fbbe55ba53,e97c86131c1d585f
COVID-93.png,COVID,COVID/images/COVID-93.png,COVID/masks/COVID-93.png,True,G00177,val,269985393b3844ce7dbd630f0156fd4e,8d75d21a671e1b45
COVID-930.png,COVID,COVID/images/COVID-930.png,COVID/masks/COVID-930.png,True,G03332,test,286e022d7f5f9cd07c33ef6bcdaa90bb,c17f06161f0f1e59
COVID-931.png,COVID,COVID/images/COVID-931.png,COVID/masks/COVID-931.png,True,G03333,test,afc8ed502acb34f39fe9ae9c5590a9d7,813d061e1a5f5f59
COVID-932.png,COVID,COVID/images/COVID-932.png,COVID/masks/COVID-932.png,True,G03334,test,9a8115779b6acde40321215c18450fdd,d53b55475e1a1a1a
COVID-933.png,COVID,COVID/images/COVID-933.png,COVID/masks/COVID-933.png,True,G03335,train,993e354f332ff2f8870fba4965ee5111,c43c138b1ad37be8
COVID-934.png,COVID,COVID/images/COVID-934.png,COVID/masks/COVID-934.png,True,G03336,train,9707ef8f81d71ca262d818bdfef649b2,8d2e93030f161f6f
COVID-935.png,COVID,COVID/images/COVID-935.png,COVID/masks/COVID-935.png,True,G03337,train,6ad57e38965db12509ddb625ee4bf927,813c861f191d597f
COVID-936.png,COVID,COVID/images/COVID-936.png,COVID/masks/COVID-936.png,True,G03338,train,31f8d5ac99156a5aff98cae4a374cdb2,d52857424f5e1a6b
COVID-937.png,COVID,COVID/images/COVID-937.png,COVID/masks/COVID-937.png,True,G03339,train,4d5519d9d82254e0e3952f1af3147dda,ad44d21b4f4a0fe3
COVID-938.png,COVID,COVID/images/COVID-938.png,COVID/masks/COVID-938.png,True,G03340,val,0cdf8cf9409a20c587c04335cf03138f,ee2d594b4b490e2d
COVID-939.png,COVID,COVID/images/COVID-939.png,COVID/masks/COVID-939.png,True,G03341,val,fab52cec32a5682dc9e3e702c03ec3cf,d43d63c3971a1b43
COVID-94.png,COVID,COVID/images/COVID-94.png,COVID/masks/COVID-94.png,True,G03342,train,0f047e602051b716917eba651337e422,9168061a5f5e595f
COVID-940.png,COVID,COVID/images/COVID-940.png,COVID/masks/COVID-940.png,True,G03343,train,1799ff8d378a535d7b52806fcb9d16e1,b925c21ac74a9f47
COVID-941.png,COVID,COVID/images/COVID-941.png,COVID/masks/COVID-941.png,True,G03344,test,448ece4261538021737709b9667e327b,893c5e5c105b397b
COVID-942.png,COVID,COVID/images/COVID-942.png,COVID/masks/COVID-942.png,True,G03345,test,d8e9cd3e2dadbfe47ab5aefab8a35f90,817f4716184f5f1c
COVID-943.png,COVID,COVID/images/COVID-943.png,COVID/masks/COVID-943.png,True,G03346,train,3dbb4f58a812765616633afc876c09c2,d869c51e4cd21eda
COVID-944.png,COVID,COVID/images/COVID-944.png,COVID/masks/COVID-944.png,True,G03347,test,d0444e7a630e728464f9e1686a67dd28,983fc71d1c4b1d1c
COVID-945.png,COVID,COVID/images/COVID-945.png,COVID/masks/COVID-945.png,True,G03348,train,1564e7143edf98ad9b855f7c3607e409,817a16186da5b5e7
COVID-946.png,COVID,COVID/images/COVID-946.png,COVID/masks/COVID-946.png,True,G00261,train,4cf1e27e21eeb674210cfd6d3fa6b91d,9c6d431e491e1f4b
COVID-947.png,COVID,COVID/images/COVID-947.png,COVID/masks/COVID-947.png,True,G03349,train,f1f8af8a4f8710109d57bf3961198be5,83a33c9578957a93
COVID-948.png,COVID,COVID/images/COVID-948.png,COVID/masks/COVID-948.png,True,G03350,train,09ff087d93b2b4c240e5ef0edde73865,817f141e1b1b1b3b
COVID-949.png,COVID,COVID/images/COVID-949.png,COVID/masks/COVID-949.png,True,G03351,train,eae927cfdffd25bd77782d4e86246aa1,8f4dc0d6cb4b5915
COVID-95.png,COVID,COVID/images/COVID-95.png,COVID/masks/COVID-95.png,True,G01472,train,08d5fe5992d89faa04951ce7857b43bb,9179061e5f5a3c5a
COVID-950.png,COVID,COVID/images/COVID-950.png,COVID/masks/COVID-950.png,True,G03352,train,b467a10a216e8e71d98a3cc7b50a064c,c1e13e872f16c62b
COVID-951.png,COVID,COVID/images/COVID-951.png,COVID/masks/COVID-951.png,True,G03353,train,d0d744c6a4aece46d522d58c7fcdc3c0,c825971ed71e7948
COVID-952.png,COVID,COVID/images/COVID-952.png,COVID/masks/COVID-952.png,True,G03354,train,7ad65a80df454bf9d80ffe1bb689f8ce,d929961e435a1e57
COVID-953.png,COVID,COVID/images/COVID-953.png,COVID/masks/COVID-953.png,True,G03355,val,7c56067e50f0dc8574e81730a69e6b76,ac2e8b0dc8ecf2f0
COVID-954.png,COVID,COVID/images/COVID-954.png,COVID/masks/COVID-954.png,True,G03356,train,8b0d92a323db17a2671d9c5b414e989c,f91916165e19595e
COVID-955.png,COVID,COVID/images/COVID-955.png,COVID/masks/COVID-955.png,True,G03357,train,3e36f1f3190cbdc0fad8f22a3d0c6869,bd2cc4939b830f1e
COVID-956.png,COVID,COVID/images/COVID-956.png,COVID/masks/COVID-956.png,True,G03358,train,9da27714ad177a029b1c805c7bf90f38,817b141e1a1b3b3f
COVID-957.png,COVID,COVID/images/COVID-957.png,COVID/masks/COVID-957.png,True,G03359,train,8658d45a6bc786b5dfee293821ee5787,f97d07161a59491d
COVID-958.png,COVID,COVID/images/COVID-958.png,COVID/masks/COVID-958.png,True,G03360,train,ea2c43ea0da4d73f3ce57e290e0ddec6,dd04431e1f495aed
COVID-959.png,COVID,COVID/images/COVID-959.png,COVID/masks/COVID-959.png,True,G03361,train,9b411c47161d8f258a2ed7805dd87051,ee3a514d5131f0f1
COVID-96.png,COVID,COVID/images/COVID-96.png,COVID/masks/COVID-96.png,True,G03362,train,5bafa6868b3eab42d8266a3e025bb2ab,9929e7161ec5391e
COVID-960.png,COVID,COVID/images/COVID-960.png,COVID/masks/COVID-960.png,True,G03363,train,bba074b9411ee4715153139a70290a54,dd3e425f144b3d28
COVID-961.png,COVID,COVID/images/COVID-961.png,COVID/masks/COVID-961.png,True,G03364,train,97c027fe6b6bfd3fb4d9f1323afd734d,fc6ce0124b585f4d
COVID-962.png,COVID,COVID/images/COVID-962.png,COVID/masks/COVID-962.png,True,G03365,train,306ba2b63086341e6e4d7410595a016f,d43a534f5b4f0a29
COVID-963.png,COVID,COVID/images/COVID-963.png,COVID/masks/COVID-963.png,True,G03366,train,10c20f881de7c375133ea6ac00bb6b14,8879871e5a0b1d5f
COVID-964.png,COVID,COVID/images/COVID-964.png,COVID/masks/COVID-964.png,True,G03367,train,236c8e9da3d637630ae45d89e154a78b,ac6dd0974f401e1f
COVID-965.png,COVID,COVID/images/COVID-965.png,COVID/masks/COVID-965.png,True,G03368,train,ef3fb6458da2a24a0731e61cba69998d,a93f960e1e091d3d
COVID-966.png,COVID,COVID/images/COVID-966.png,COVID/masks/COVID-966.png,True,G03369,val,32fa3083bd536f52dcf8e38362c08fc6,cd4d56161c6b496d
COVID-967.png,COVID,COVID/images/COVID-967.png,COVID/masks/COVID-967.png,True,G03370,val,cd623988ea97637d919e01e6606aa85d,bd6cc61645451e4f
COVID-968.png,COVID,COVID/images/COVID-968.png,COVID/masks/COVID-968.png,True,G03371,val,bc46bd2b1a5b76cd643590e4e40ab152,ac3ed3dfc6c24c40
COVID-969.png,COVID,COVID/images/COVID-969.png,COVID/masks/COVID-969.png,True,G03372,train,515005a7f4b4d95ca85737ec6597ca21,892b8e161e0e3f3b
COVID-97.png,COVID,COVID/images/COVID-97.png,COVID/masks/COVID-97.png,True,G00002,train,c6bdddf9cb6c43ea52fe73aa6b45664b,992a061a4f79794f
COVID-970.png,COVID,COVID/images/COVID-970.png,COVID/masks/COVID-970.png,True,G03373,test,7fbfb28dad702b5b0623a0dea8246fc9,9c2cc31b0b4f1e4f
COVID-971.png,COVID,COVID/images/COVID-971.png,COVID/masks/COVID-971.png,True,G03374,test,43201de572d501f73e544d2090956c90,d42f034a3a7a3cf4
COVID-972.png,COVID,COVID/images/COVID-972.png,COVID/masks/COVID-972.png,True,G03375,train,930a9e0963c4d4eb8c84af285ff84868,9527d1810f4d2e77
COVID-973.png,COVID,COVID/images/COVID-973.png,COVID/masks/COVID-973.png,True,G03376,train,8aff4faa8f7ff1c426499ff557041bc7,cd2dd29097c56a93
COVID-974.png,COVID,COVID/images/COVID-974.png,COVID/masks/COVID-974.png,True,G03377,test,97ecfbdd5feb79a955af75e4e37275a2,d52607430d476b5f
COVID-975.png,COVID,COVID/images/COVID-975.png,COVID/masks/COVID-975.png,True,G03378,train,1a33137e3046e246f391e27a5881351c,d951d7565a78505a
COVID-976.png,COVID,COVID/images/COVID-976.png,COVID/masks/COVID-976.png,True,G03379,train,e9974a5cd1e90da93b0fc72faa8a94cb,990c063b0d3f2d6f
COVID-977.png,COVID,COVID/images/COVID-977.png,COVID/masks/COVID-977.png,True,G03380,train,321163790061b78fbcb14fb8d87066a2,f859c1164fc29e4b
COVID-978.png,COVID,COVID/images/COVID-978.png,COVID/masks/COVID-978.png,True,G03381,train,4d0c09f5444611ba97da9ebf34dd9dfa,99fe861b47114b96
COVID-979.png,COVID,COVID/images/COVID-979.png,COVID/masks/COVID-979.png,True,G03382,train,2fb54a975b4b158bb7fbf5da8fd5169f,837804373e7a7672
COVID-98.png,COVID,COVID/images/COVID-98.png,COVID/masks/COVID-98.png,True,G00847,train,7cc16da40e14f6a095746e9c39ba85b3,9d6cc01b4f4d1a5d
COVID-980.png,COVID,COVID/images/COVID-980.png,COVID/masks/COVID-980.png,True,G03383,train,0195a5011dd3e3832adb050ad633d888,882fc2d33fc06d5c
COVID-981.png,COVID,COVID/images/COVID-981.png,COVID/masks/COVID-981.png,True,G03384,train,30922175c44aa2143edf4699e5f5ccd2,cd24d15a831b9f1d
COVID-982.png,COVID,COVID/images/COVID-982.png,COVID/masks/COVID-982.png,True,G03385,val,98ac3aa2ea3e53d249cf9d59f93737ee,d42b531d4a5f2a4b
COVID-983.png,COVID,COVID/images/COVID-983.png,COVID/masks/COVID-983.png,True,G03386,train,8af7c1911156c6c6dd25d52b14214caa,817f780511171f5f
COVID-984.png,COVID,COVID/images/COVID-984.png,COVID/masks/COVID-984.png,True,G03387,train,890ae2cb2da2c833543ef5945c255a59,ed6cd21a0d4a1bcd
COVID-985.png,COVID,COVID/images/COVID-985.png,COVID/masks/COVID-985.png,True,G03388,val,f742aad430963bbd60ec48b701851125,dc2e825b2fc23dc1
COVID-986.png,COVID,COVID/images/COVID-986.png,COVID/masks/COVID-986.png,True,G03389,test,f5f2688c63fe3efef9b2c71607d23a72,d52a43531f4e4c4f
COVID-987.png,COVID,COVID/images/COVID-987.png,COVID/masks/COVID-987.png,True,G03390,train,9a858585c1d99b6b3aa839c0f09a01aa,dd0c421e4dcdd8f8
COVID-988.png,COVID,COVID/images/COVID-988.png,COVID/masks/COVID-988.png,True,G03391,train,4d3d7695dd161331a483b2d459130add,c02d4edb1b8d2f98
COVID-989.png,COVID,COVID/images/COVID-989.png,COVID/masks/COVID-989.png,True,G03392,train,bdd41dc2dc6b1fd217b517c835068f95,837a14151f5f1e5a
COVID-99.png,COVID,COVID/images/COVID-99.png,COVID/masks/COVID-99.png,True,G03393,val,1e3649e12b474829cfd7b37f903bdf60,9d2de5121fa3921e
COVID-990.png,COVID,COVID/images/COVID-990.png,COVID/masks/COVID-990.png,True,G03394,test,afef8b168b86058e156604356911bee8,dd69421743470fcb
COVID-991.png,COVID,COVID/images/COVID-991.png,COVID/masks/COVID-991.png,True,G03395,test,21e711eecf266df418f151b634b06b5e,857886021f0f3f3f
COVID-992.png,COVID,COVID/images/COVID-992.png,COVID/masks/COVID-992.png,True,G03396,train,c84c341c25642e1c3540441520480369,85581f7e1e587869
COVID-993.png,COVID,COVID/images/COVID-993.png,COVID/masks/COVID-993.png,True,G02222,train,665cdec947e2cc0a2f2b62b6f2bfc3f7,c92df4920fe55c1a
COVID-994.png,COVID,COVID/images/COVID-994.png,COVID/masks/COVID-994.png,True,G03397,train,137e0d08854accc1638159fd57ea95a7,817d87821e4f1f3a
COVID-995.png,COVID,COVID/images/COVID-995.png,COVID/masks/COVID-995.png,True,G03297,train,21b0c8c4f954e91a0518963346e45131,813d0c1c1b1f5f79
COVID-996.png,COVID,COVID/images/COVID-996.png,COVID/masks/COVID-996.png,True,G03398,train,8c01064cbf44903923b25a63018f9aff,d875d20bd55a4758
COVID-997.png,COVID,COVID/images/COVID-997.png,COVID/masks/COVID-997.png,True,G03399,train,1aa81932fdd81441d288368610d540bc,c07dd6461a5f8a1b
COVID-998.png,COVID,COVID/images/COVID-998.png,COVID/masks/COVID-998.png,True,G03400,train,25e8ac5547dd469454d424987a124fe6,c4192f1b3b24b576
COVID-999.png,COVID,COVID/images/COVID-999.png,COVID/masks/COVID-999.png,True,G03401,train,6989f277b0772f6069b8082bc834b8d5,fc2dc012c7c61fc3
Lung_Opacity-1.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1.png,Lung_Opacity/masks/Lung_Opacity-1.png,True,G03402,train,10b9264097281fe68d48259eb5e9cfd7,9c2ec31fc84a3e6c
Lung_Opacity-10.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-10.png,Lung_Opacity/masks/Lung_Opacity-10.png,True,G03403,train,f626b5dc6f37bd8fd1bd80ca94b08c85,802b5a1b0a5f1f3f
Lung_Opacity-100.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-100.png,Lung_Opacity/masks/Lung_Opacity-100.png,True,G03404,train,6d0e39230cc9ff46cbfe012d056ff505,c43dd3434f0b6b68
Lung_Opacity-1000.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1000.png,Lung_Opacity/masks/Lung_Opacity-1000.png,True,G03405,train,32d94800ecbc8b302a0fc3ceace2d24b,802a5f1a5a1b5f2f
Lung_Opacity-1001.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1001.png,Lung_Opacity/masks/Lung_Opacity-1001.png,True,G03406,train,35cc90abf7d3b748b67769f2f3b198e0,807bd65ed0565a59
Lung_Opacity-1002.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1002.png,Lung_Opacity/masks/Lung_Opacity-1002.png,True,G03407,val,9b5fcf2103ea80eeb965de08d7ee4eed,cc2d434f4c2e0f8f
Lung_Opacity-1003.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1003.png,Lung_Opacity/masks/Lung_Opacity-1003.png,True,G03408,train,56d110747a17b46c5cb97ae0cb095c7a,d52a030f575e1d4e
Lung_Opacity-1004.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1004.png,Lung_Opacity/masks/Lung_Opacity-1004.png,True,G03409,val,d07407783557ed1fe9a3f58f7446a713,983fc21a1d6b1a2f
Lung_Opacity-1005.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1005.png,Lung_Opacity/masks/Lung_Opacity-1005.png,True,G03410,train,eb505071a48fe0903b69c18eccae9212,9b78161e0b3a2b3b
Lung_Opacity-1006.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1006.png,Lung_Opacity/masks/Lung_Opacity-1006.png,True,G03411,val,39cf8b352d50a41cd28abedfb5275384,dc2d420b0f4e6e7c
Lung_Opacity-1007.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1007.png,Lung_Opacity/masks/Lung_Opacity-1007.png,True,G03412,test,31ba809ce545a7941f4355034066470b,d06bc61e1b0b0f4f
Lung_Opacity-1008.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1008.png,Lung_Opacity/masks/Lung_Opacity-1008.png,True,G03413,train,1e0520629a5ea41a424f31a59a05fee2,916807de1a5e3ada
Lung_Opacity-1009.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1009.png,Lung_Opacity/masks/Lung_Opacity-1009.png,True,G03414,train,de7ad373ead9f9c6607c514ecec5dd3d,913bc68e1d8a5f19
Lung_Opacity-101.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-101.png,Lung_Opacity/masks/Lung_Opacity-101.png,True,G03415,test,a96117e8501841740cf80c63fde3c9f2,c42f930b4d0f1f1e
Lung_Opacity-1010.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1010.png,Lung_Opacity/masks/Lung_Opacity-1010.png,True,G03416,train,9b160743bb519681a0ac62159ea6d18d,917906861e1f1b5f
Lung_Opacity-1011.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1011.png,Lung_Opacity/masks/Lung_Opacity-1011.png,True,G03417,test,f1b35820dcf4d95584b3a03687f6d7fe,8178165e7b686567
Lung_Opacity-1012.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1012.png,Lung_Opacity/masks/Lung_Opacity-1012.png,True,G03418,train,8f49ce78c79ee828348bf36cc46c1369,8d681f5e58597949
Lung_Opacity-1013.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1013.png,Lung_Opacity/masks/Lung_Opacity-1013.png,True,G03419,train,133b7cfbbf037778542cc6fd2a4ff16a,807f061a1f4f585f
Lung_Opacity-1014.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1014.png,Lung_Opacity/masks/Lung_Opacity-1014.png,True,G03420,train,f0a2632c2e404e98982d7cba27326372,d13b061a1d4f7939
Lung_Opacity-1015.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1015.png,Lung_Opacity/masks/Lung_Opacity-1015.png,True,G03421,train,b0c788eba1cf7c2ef8e8fb20481e3efd,c825c61bc71cb94f
Lung_Opacity-1016.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1016.png,Lung_Opacity/masks/Lung_Opacity-1016.png,True,G03422,train,16fdd0c5f3e3dfb0cc53b6a6fb5f93c6,852a505b597f7a31
Lung_Opacity-1017.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1017.png,Lung_Opacity/masks/Lung_Opacity-1017.png,True,G03423,train,ad43626381d22e52b31463de075fd3d9,9178561919293bbf
Lung_Opacity-1018.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1018.png,Lung_Opacity/masks/Lung_Opacity-1018.png,True,G03424,train,624b81c8b34f0079a454f18b24d63840,d62f450f0e4a0f4f
Lung_Opacity-1019.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1019.png,Lung_Opacity/masks/Lung_Opacity-1019.png,True,G03425,train,bbbb5af0a1ddcfec5e490db58d915333,857c0e070f0f1f1f
Lung_Opacity-102.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-102.png,Lung_Opacity/masks/Lung_Opacity-102.png,True,G03426,train,5de24ceaec8181624ca9550622ee7717,942f46de18784dec
Lung_Opacity-1020.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1020.png,Lung_Opacity/masks/Lung_Opacity-1020.png,True,G03427,train,b542b77ce319ec369a277088862b103c,83699c7a7b616361
Lung_Opacity-1021.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1021.png,Lung_Opacity/masks/Lung_Opacity-1021.png,True,G03428,train,ce7712a0c15e552dd2cf5c7a61720137,807b56d6581b5b2b
Lung_Opacity-1022.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1022.png,Lung_Opacity/masks/Lung_Opacity-1022.png,True,G03429,val,bea7cd1db880c3815d41f4612a4841fe,dd2a1a560a4f0f1f
Lung_Opacity-1023.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1023.png,Lung_Opacity/masks/Lung_Opacity-1023.png,True,G03430,test,979c45aef69ee01be38886e51ed81c0e,d13906965e1d4b5b
Lung_Opacity-1024.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1024.png,Lung_Opacity/masks/Lung_Opacity-1024.png,True,G03431,val,46726300ed7c85fae438ca7d72deba60,c03dd7c2875d3d0c
Lung_Opacity-1025.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1025.png,Lung_Opacity/masks/Lung_Opacity-1025.png,True,G03432,train,58852638a9ab06861072bddedc13f081,917b07469e1eba51
Lung_Opacity-1026.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1026.png,Lung_Opacity/masks/Lung_Opacity-1026.png,True,G03433,test,abb4d9ecb0822adfea424046dd9b8948,c47f93510b5e4a4b
Lung_Opacity-1027.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1027.png,Lung_Opacity/masks/Lung_Opacity-1027.png,True,G03434,val,219a6a5c0c6f23f9d49a2130851aaa56,c46dc34a4d4f2e4d
Lung_Opacity-1028.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1028.png,Lung_Opacity/masks/Lung_Opacity-1028.png,True,G03435,test,e2827ed5e674aa063f12cc19cb423179,cc3d4a0d2c6ce9f1
Lung_Opacity-1029.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1029.png,Lung_Opacity/masks/Lung_Opacity-1029.png,True,G03436,train,a7c8bf16f334fd907d2a5f40a6c7c1a3,d02b4f047636e667
Lung_Opacity-103.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-103.png,Lung_Opacity/masks/Lung_Opacity-103.png,True,G03437,train,7f5cc963647b75746400b4e857ffa488,c562c34f686dac74
Lung_Opacity-1030.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1030.png,Lung_Opacity/masks/Lung_Opacity-1030.png,True,G03438,val,e5f7512688126dc1c692150991a207d6,c02f7f024b696d59
Lung_Opacity-1031.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1031.png,Lung_Opacity/masks/Lung_Opacity-1031.png,True,G03439,train,21420e3f86412927390a1b4b368a74e3,916a071616575b5f
Lung_Opacity-1032.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1032.png,Lung_Opacity/masks/Lung_Opacity-1032.png,True,G03440,train,c573e08bf7d23c0a091b3d69858943b4,dd6a875a4d4c4c4d
Lung_Opacity-1033.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1033.png,Lung_Opacity/masks/Lung_Opacity-1033.png,True,G03441,test,d60c0b4774e1d2dfd039bc451594185a,817846961e1d5f5d
Lung_Opacity-1034.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1034.png,Lung_Opacity/masks/Lung_Opacity-1034.png,True,G03442,train,064f4c6337e24329481ff59706058f56,c179171a3a3b3939
Lung_Opacity-1035.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1035.png,Lung_Opacity/masks/Lung_Opacity-1035.png,True,G03443,train,672ae00ed903c5435bb2cad99db09a85,907bc7071c1d571b
Lung_Opacity-1036.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1036.png,Lung_Opacity/masks/Lung_Opacity-1036.png,True,G03444,train,6759eef2ec48f59d0adaa789ca46cd08,c03d4e0e0e0f6f6e
Lung_Opacity-1037.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1037.png,Lung_Opacity/masks/Lung_Opacity-1037.png,True,G03445,train,d3a2013a322c1e7fdcd561a3325f9fb7,d5280e5e1a7a6a7a
Lung_Opacity-1038.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1038.png,Lung_Opacity/masks/Lung_Opacity-1038.png,True,G03446,train,7560c3a2099e61d80af2b89beb7da37b,d1efc6181c5a195b
Lung_Opacity-1039.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1039.png,Lung_Opacity/masks/Lung_Opacity-1039.png,True,G03447,train,37b01cde4c5f36e5d27fdd17b7397e85,d42e1f1d1c0e6c4f
Lung_Opacity-104.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-104.png,Lung_Opacity/masks/Lung_Opacity-104.png,True,G03448,train,b52b6934e3de71e8ce3ea66daf687f3a,c13d62d25e5ed4c1
Lung_Opacity-1040.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1040.png,Lung_Opacity/masks/Lung_Opacity-1040.png,True,G03449,train,8b6de64bdb6ca2d361812408300e91b6,903f464e594be9e8
Lung_Opacity-1041.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1041.png,Lung_Opacity/masks/Lung_Opacity-1041.png,True,G03450,train,cd2993f6ea934023da6052d5884f6f68,cc2c52534f4b4e4f
Lung_Opacity-1042.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1042.png,Lung_Opacity/masks/Lung_Opacity-1042.png,True,G03451,test,fb627f784e63f13d9e5c3cf094c269ca,9968865e597d5878
Lung_Opacity-1043.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1043.png,Lung_Opacity/masks/Lung_Opacity-1043.png,True,G03452,val,4e897b3cfffa6754a9bd663c41f1b946,8978165a597a69ea
Lung_Opacity-1044.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1044.png,Lung_Opacity/masks/Lung_Opacity-1044.png,True,G03453,val,2e480bc9e674b24c488a11a4bf0429dd,c93d461919ae39b5
Lung_Opacity-1045.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1045.png,Lung_Opacity/masks/Lung_Opacity-1045.png,True,G03454,train,bb112cc5cc138fb5053b163495fa49aa,956cc61a4d5e1c5e
Lung_Opacity-1046.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1046.png,Lung_Opacity/masks/Lung_Opacity-1046.png,True,G03455,test,3b069d25feded8f1b9905d9ec332ba92,c42d52131d784b7f
Lung_Opacity-1047.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1047.png,Lung_Opacity/masks/Lung_Opacity-1047.png,True,G03456,train,6e2f8d9cfb3ef343141a0b980e0e2cfc,837815175a73634f
Lung_Opacity-1048.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1048.png,Lung_Opacity/masks/Lung_Opacity-1048.png,True,G03457,test,b4ab6722d7e5ee9019039f5297c3550b,d929d7161e4a591d
Lung_Opacity-1049.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1049.png,Lung_Opacity/masks/Lung_Opacity-1049.png,True,G03458,train,450718b94d1ebee376a13ccedcccfb52,d12bd2c35f18cc4b
Lung_Opacity-105.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-105.png,Lung_Opacity/masks/Lung_Opacity-105.png,True,G03459,train,d751fd5edef97caed740093eabe1afa0,c91f165a5b19393a
Lung_Opacity-1050.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1050.png,Lung_Opacity/masks/Lung_Opacity-1050.png,True,G03460,train,82d7eb80e4dad6557e4f8ba59f0fa12e,dd38d31b0d1b0b3c
Lung_Opacity-1051.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1051.png,Lung_Opacity/masks/Lung_Opacity-1051.png,True,G03461,train,36e373483487bdba217d58ef67f8b39a,983dc6931b470d4f
Lung_Opacity-1052.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1052.png,Lung_Opacity/masks/Lung_Opacity-1052.png,True,G03462,train,e75d771ae451dc01697d1d35dd9d5bec,917e501b7f7e202c
Lung_Opacity-1053.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1053.png,Lung_Opacity/masks/Lung_Opacity-1053.png,True,G03463,train,3d57212ef57643a3ce75d8405ae869e6,917e0e185f5f5858
Lung_Opacity-1054.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1054.png,Lung_Opacity/masks/Lung_Opacity-1054.png,True,G03464,train,a15e79bc5ffad65c6170f26aa8cb6453,9978531b481f3c6b
Lung_Opacity-1055.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1055.png,Lung_Opacity/masks/Lung_Opacity-1055.png,True,G03465,train,9719b816d2c641ea6d2e1192c9087ac8,d43d4a4b6e2e2e0e
Lung_Opacity-1056.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1056.png,Lung_Opacity/masks/Lung_Opacity-1056.png,True,G03466,train,5002f4e272575131712f3f62cbe04041,817c84045fde787b
Lung_Opacity-1057.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1057.png,Lung_Opacity/masks/Lung_Opacity-1057.png,True,G03467,val,0728ea2ac29f3a18fde1c5eb6d0955b6,c03d170f0a6f6c6e
Lung_Opacity-1058.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1058.png,Lung_Opacity/masks/Lung_Opacity-1058.png,True,G03468,train,9574480dfa33c1ca6e7c09a02bf549ab,c929179b59093cbb
Lung_Opacity-1059.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1059.png,Lung_Opacity/masks/Lung_Opacity-1059.png,True,G03469,train,e543633e58d7ee64fca1869239b8edb3,95795a1a5d9c3896
Lung_Opacity-106.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-106.png,Lung_Opacity/masks/Lung_Opacity-106.png,True,G03470,train,387506c691823c828a487b10355cc731,83600d1e1f7a5f59
Lung_Opacity-1060.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1060.png,Lung_Opacity/masks/Lung_Opacity-1060.png,True,G03471,val,fb5a5ba8793d4a8dcb3e66ca0e310d18,c43f4d17b3736160
Lung_Opacity-1061.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1061.png,Lung_Opacity/masks/Lung_Opacity-1061.png,True,G03472,train,3d6af5151931b2f24078b6063a476069,d821d70f915f885e
Lung_Opacity-1062.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1062.png,Lung_Opacity/masks/Lung_Opacity-1062.png,True,G03473,test,6b59f03f0169d23f746cc0aa0c2c929c,c93e47121e155b1f
Lung_Opacity-1063.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1063.png,Lung_Opacity/masks/Lung_Opacity-1063.png,True,G03474,val,b99b2e8ba26e0044f1a3b04984b7324c,907f16181b3b1b2f
Lung_Opacity-1064.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1064.png,Lung_Opacity/masks/Lung_Opacity-1064.png,True,G03475,test,226a013d8df294bf5c8b23b2d6e1b523,d8794787160e1f0f
Lung_Opacity-1065.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1065.png,Lung_Opacity/masks/Lung_Opacity-1065.png,True,G03476,train,a2f0d2a9fca8fedf3d59ee42b71864b7,c52e1716585d5f49
Lung_Opacity-1066.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1066.png,Lung_Opacity/masks/Lung_Opacity-1066.png,True,G03477,train,8c8345fba9ad0ba3ee1d12c4daabf222,80fd4703934f1f0f
Lung_Opacity-1067.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1067.png,Lung_Opacity/masks/Lung_Opacity-1067.png,True,G03478,train,21499c4f3ac9112061d7968f8957b285,803adf195f1a533a
Lung_Opacity-1068.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1068.png,Lung_Opacity/masks/Lung_Opacity-1068.png,True,G03479,train,30ce452f71626f3745f5ba67938d3ec2,cc2dd3c31d0c4e4f
Lung_Opacity-1069.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1069.png,Lung_Opacity/masks/Lung_Opacity-1069.png,True,G03480,train,2dce42844733d0bcf57e6d915df82deb,d17b52475c195d0b
Lung_Opacity-107.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-107.png,Lung_Opacity/masks/Lung_Opacity-107.png,True,G03481,train,78f8a68c069bf1cd9aa4aa3a423f0b5f,c03d47431b1ced6d
Lung_Opacity-1070.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1070.png,Lung_Opacity/masks/Lung_Opacity-1070.png,True,G03482,train,0c0ae5f05014f6446bed9fd3b74d740b,d13fe2464f444e2b
Lung_Opacity-1071.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1071.png,Lung_Opacity/masks/Lung_Opacity-1071.png,True,G03483,train,c00ee6ac779e50cb302b269045938eee,c03db3530f4eb05e
Lung_Opacity-1072.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1072.png,Lung_Opacity/masks/Lung_Opacity-1072.png,True,G03484,train,05c32a67e90ba58c5ca03bada25db0be,d42f42430f4e4f1f
Lung_Opacity-1073.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1073.png,Lung_Opacity/masks/Lung_Opacity-1073.png,True,G03485,train,cbb35b68c282b3d1ba6ccc072d7aee7e,95681e86185fcf78
Lung_Opacity-1074.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1074.png,Lung_Opacity/masks/Lung_Opacity-1074.png,True,G03486,train,e685dc81cea79a7865c288d980b7deb0,c725400e6b607dfd
Lung_Opacity-1075.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1075.png,Lung_Opacity/masks/Lung_Opacity-1075.png,True,G03487,val,85a0a0ddaee4dc31893c93ca52e684dc,ca35c90d8fb59594
Lung_Opacity-1076.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1076.png,Lung_Opacity/masks/Lung_Opacity-1076.png,True,G03488,test,9a257e29907a088262b3ab12b260c6ca,d62f422e266ae4f1
Lung_Opacity-1077.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1077.png,Lung_Opacity/masks/Lung_Opacity-1077.png,True,G03489,val,d8166b17da578d2a909b8d2882abb27e,d03f96c24b5f5c48
Lung_Opacity-1078.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1078.png,Lung_Opacity/masks/Lung_Opacity-1078.png,True,G03490,test,13dc176d51b866d725335dcd5acd751f,857e40159e5e5c5d
Lung_Opacity-1079.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1079.png,Lung_Opacity/masks/Lung_Opacity-1079.png,True,G03491,train,d2532a258f481c9e7010d7249ba4147e,916842905bdf699f
Lung_Opacity-108.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-108.png,Lung_Opacity/masks/Lung_Opacity-108.png,True,G03492,train,b13a1c6a00c20a85d3a2684ce02a6e59,916a061e5e5a5d5d
Lung_Opacity-1080.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1080.png,Lung_Opacity/masks/Lung_Opacity-1080.png,True,G03493,train,3edc7b3ee6684c77138781ee1bef09fc,883bd78f1d189317
Lung_Opacity-1081.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1081.png,Lung_Opacity/masks/Lung_Opacity-1081.png,True,G03494,train,b97eedb74e10d886e322d8d0b76cf410,817f5759481d2d3a
Lung_Opacity-1082.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1082.png,Lung_Opacity/masks/Lung_Opacity-1082.png,True,G03495,train,cc1a548d4434854daa6c33f9e378cee7,c03f2f4585033e3f
Lung_Opacity-1083.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1083.png,Lung_Opacity/masks/Lung_Opacity-1083.png,True,G03496,train,1e04b089cc37ae007ea343d0653aad18,8179f2512b71aa76
Lung_Opacity-1084.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1084.png,Lung_Opacity/masks/Lung_Opacity-1084.png,True,G03497,val,73858d3a6fb5b31ff7630648f9f68339,cc3dd3c1434d6e58
Lung_Opacity-1085.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1085.png,Lung_Opacity/masks/Lung_Opacity-1085.png,True,G03498,test,e3d0c2b7f6ea551afdad2588015fd6cb,942f471f4e686a63
Lung_Opacity-1086.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1086.png,Lung_Opacity/masks/Lung_Opacity-1086.png,True,G03499,train,bc285af970b368fd54cd4d445437e401,dc28c3c3575f582a
Lung_Opacity-1087.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1087.png,Lung_Opacity/masks/Lung_Opacity-1087.png,True,G03500,train,e4aae84b828f5552e75c8b24047d4b90,cc7fc7493c6d0c0a
Lung_Opacity-1088.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1088.png,Lung_Opacity/masks/Lung_Opacity-1088.png,True,G03501,train,b069c12bb9ae004539f4fdf06667330c,dd3b9618d1e2e4e0
Lung_Opacity-1089.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1089.png,Lung_Opacity/masks/Lung_Opacity-1089.png,True,G03502,train,bb218ae2f432bc4f1f1f83e560ed27d3,83723d9d155b3b11
Lung_Opacity-109.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-109.png,Lung_Opacity/masks/Lung_Opacity-109.png,True,G03503,train,3aa26105d79dfe30ae2b4c04b3a49542,c0295752574a5f5b
Lung_Opacity-1090.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1090.png,Lung_Opacity/masks/Lung_Opacity-1090.png,True,G03504,train,eeb3c5a3e9be81b76f84b3cf25f95648,813fc69a8bc73d0a
Lung_Opacity-1091.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1091.png,Lung_Opacity/masks/Lung_Opacity-1091.png,True,G03505,train,6e85ec3d184d4ab40075dcef3e27c9f3,c42f430f2e2e2e7a
Lung_Opacity-1092.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1092.png,Lung_Opacity/masks/Lung_Opacity-1092.png,True,G03506,train,0fbb02f74e8a39f3939af6fea80fa12a,9568075e5b786b62
Lung_Opacity-1093.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1093.png,Lung_Opacity/masks/Lung_Opacity-1093.png,True,G03507,train,7296fa146f77887d080996beb70d635e,807fa6861e3d1f0d
Lung_Opacity-1094.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1094.png,Lung_Opacity/masks/Lung_Opacity-1094.png,True,G03508,train,9728c71f7823ec1b664c1f7fe9119be2,d46b830d055f0f4f
Lung_Opacity-1095.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1095.png,Lung_Opacity/masks/Lung_Opacity-1095.png,True,G03509,train,29a093857d4c3267b195d00055f41a28,916e8712cb71a95d
Lung_Opacity-1096.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1096.png,Lung_Opacity/masks/Lung_Opacity-1096.png,True,G03510,val,4f36c99cc0d597b195038e407d4ee041,995d101e5a1f5e2e
Lung_Opacity-1097.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1097.png,Lung_Opacity/masks/Lung_Opacity-1097.png,True,G03511,val,d50dd8f10150c49979b78d7f420090ac,881f061a1f587b7e
Lung_Opacity-1098.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1098.png,Lung_Opacity/masks/Lung_Opacity-1098.png,True,G03512,test,093a523c93d54ecee9bd8d91a5de0b1d,c91d144a1d7b3a6b
Lung_Opacity-1099.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1099.png,Lung_Opacity/masks/Lung_Opacity-1099.png,True,G03513,val,d6bb2f833b3dbf9a8de75c2be7c0c4f1,817d141e5a5c7c7a
Lung_Opacity-11.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-11.png,Lung_Opacity/masks/Lung_Opacity-11.png,True,G03514,test,0346617aab11b6cb8fc1c22167d9c360,9978e50fd4821bf8
Lung_Opacity-110.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-110.png,Lung_Opacity/masks/Lung_Opacity-110.png,True,G03515,test,ab27a0884acee29f5cb266dc8f6e81e6,917b46161a4f5f58
Lung_Opacity-1100.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1100.png,Lung_Opacity/masks/Lung_Opacity-1100.png,True,G03516,train,18e9af23e742a7f09a02542c278acd49,823d78a52d6267e3
Lung_Opacity-1101.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1101.png,Lung_Opacity/masks/Lung_Opacity-1101.png,True,G03517,val,fbe22f27532a47664a4bbaa47fb40f74,c02d3fd235f1349a
Lung_Opacity-1102.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1102.png,Lung_Opacity/masks/Lung_Opacity-1102.png,True,G03518,train,a0acd1c9cb2a7ef42067cce47998189a,dc3cc34f420f0f4d
Lung_Opacity-1103.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1103.png,Lung_Opacity/masks/Lung_Opacity-1103.png,True,G03519,train,ea10d316e368d4d3b3017e4b21ce20fb,9d6946164b0d3df8
Lung_Opacity-1104.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1104.png,Lung_Opacity/masks/Lung_Opacity-1104.png,True,G03520,train,5c58e49d1db81754bdcf65185846a348,81681f165a5eda5b
Lung_Opacity-1105.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1105.png,Lung_Opacity/masks/Lung_Opacity-1105.png,True,G03521,train,bd8cc82f874e4dda9af56a2544bf80a2,d56a4f0c0c475f0f
Lung_Opacity-1106.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1106.png,Lung_Opacity/masks/Lung_Opacity-1106.png,True,G03522,train,1b5f8467b97f654c99fabfd0c3713562,896d06d6935b585d
Lung_Opacity-1107.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1107.png,Lung_Opacity/masks/Lung_Opacity-1107.png,True,G03523,train,004544f1db7baa903258390d4d967aed,c43b3b6693472599
Lung_Opacity-1108.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1108.png,Lung_Opacity/masks/Lung_Opacity-1108.png,True,G03524,train,32451299f29788017eed98dd4b1ab77e,d929c71a1b416a7d
Lung_Opacity-1109.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1109.png,Lung_Opacity/masks/Lung_Opacity-1109.png,True,G03525,train,f7b7d17db54bfa62427ba7e647c28d44,d03f101a3e3bb4e5
Lung_Opacity-111.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-111.png,Lung_Opacity/masks/Lung_Opacity-111.png,True,G03526,train,d674ca3cabc40345dcc309e3d803bd49,cc2ad34f4f494e1a
Lung_Opacity-1110.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1110.png,Lung_Opacity/masks/Lung_Opacity-1110.png,True,G03527,val,6aebfe615dd7793d7eb4af6321f021f9,995986165f5f1a58
Lung_Opacity-1111.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1111.png,Lung_Opacity/masks/Lung_Opacity-1111.png,True,G03528,test,2cf81a34a384501753e30458373744cc,917f461a1b4b3939
Lung_Opacity-1112.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1112.png,Lung_Opacity/masks/Lung_Opacity-1112.png,True,G03529,train,5d9bb24735d9fe2976715a6e0cb0407f,dd3ad24b5a580d5a
Lung_Opacity-1113.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1113.png,Lung_Opacity/masks/Lung_Opacity-1113.png,True,G03530,train,53ce04d93fc0183e45fe8c5ae63444e3,846c173b6b653971
Lung_Opacity-1114.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1114.png,Lung_Opacity/masks/Lung_Opacity-1114.png,True,G03531,train,24b2d4359c2d2ff23e72e9b09fba7e6d,c22e264fc566f253
Lung_Opacity-1115.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1115.png,Lung_Opacity/masks/Lung_Opacity-1115.png,True,G03532,train,4c8b4ada1b4204f2907589e2fc3fb30b,d4ae2bd16bc02fc4
Lung_Opacity-1116.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1116.png,Lung_Opacity/masks/Lung_Opacity-1116.png,True,G03533,train,ad8f9339a82fd400befbae81803a9bb5,893e161a1c595f6d
Lung_Opacity-1117.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1117.png,Lung_Opacity/masks/Lung_Opacity-1117.png,True,G03534,val,3ac9614b45a7645340656d0ec8f68a15,d03f12101f3f1f3a
Lung_Opacity-1118.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1118.png,Lung_Opacity/masks/Lung_Opacity-1118.png,True,G03535,val,be6f3b0eb0c67c1ad3844036ba852548,cc3d42434b4f0e6f
Lung_Opacity-1119.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1119.png,Lung_Opacity/masks/Lung_Opacity-1119.png,True,G03536,test,1f4c6919713b6447022615a1b57b67be,d52c42024b4f5f3f
Lung_Opacity-112.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-112.png,Lung_Opacity/masks/Lung_Opacity-112.png,True,G03537,test,10fdbdd5771ee7f8fdc0c5bce65f1775,c52d33c217c33fc8
Lung_Opacity-1120.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1120.png,Lung_Opacity/masks/Lung_Opacity-1120.png,True,G03538,train,08db8ea7971739d08c357692d06a49fa,886f071b184f4f5d
Lung_Opacity-1121.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1121.png,Lung_Opacity/masks/Lung_Opacity-1121.png,True,G03539,train,af6c2282a6e2a7ab52ae971ea5115c1a,d62f0762494b4f0f
Lung_Opacity-1122.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1122.png,Lung_Opacity/masks/Lung_Opacity-1122.png,True,G03540,train,b88492e9bc9f147cb502811754238635,817ad61c1e1a3b3b
Lung_Opacity-1123.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1123.png,Lung_Opacity/masks/Lung_Opacity-1123.png,True,G03541,train,8c646d9888ade3379dae1c3757967091,9c2d430b4c6d1e7e
Lung_Opacity-1124.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1124.png,Lung_Opacity/masks/Lung_Opacity-1124.png,True,G03542,train,2a0601dd2bcc9a7c7851c950097a1831,de254a4b4d2d2d2d
Lung_Opacity-1125.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1125.png,Lung_Opacity/masks/Lung_Opacity-1125.png,True,G03543,train,335e78a05f5ec4619b4b91825ade5eeb,dc2f5b1c2c283da9
Lung_Opacity-1126.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1126.png,Lung_Opacity/masks/Lung_Opacity-1126.png,True,G03544,train,fc5aa5bfca22faab9346142eb4604361,d42e41434f4f4b4f
Lung_Opacity-1127.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1127.png,Lung_Opacity/masks/Lung_Opacity-1127.png,True,G03545,train,0119f3dc11841c1cc0d406728aefa063,906e477e7a7a2232
Lung_Opacity-1128.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1128.png,Lung_Opacity/masks/Lung_Opacity-1128.png,True,G03546,train,f0c740e84206ef5394f21d45bccbed6b,c17f141e1b1b1b2b
Lung_Opacity-1129.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1129.png,Lung_Opacity/masks/Lung_Opacity-1129.png,True,G03547,train,f1ac527c8f665ae4f37276db33997632,c52a53492d3e3c3e
Lung_Opacity-113.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-113.png,Lung_Opacity/masks/Lung_Opacity-113.png,True,G03548,train,641002240412a4664d7f5eaabb938543,817b1e1e3b796164
Lung_Opacity-1130.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1130.png,Lung_Opacity/masks/Lung_Opacity-1130.png,True,G03549,train,61bc311b2a34f07a97850d7f64a60d10,89588e5e596a6a4f
Lung_Opacity-1131.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1131.png,Lung_Opacity/masks/Lung_Opacity-1131.png,True,G03550,train,0dd03a2a256ceed0cb311da8eadae1b1,9370841d15577a3f
Lung_Opacity-1132.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1132.png,Lung_Opacity/masks/Lung_Opacity-1132.png,True,G03551,train,f029ef5ba678739dfde98a7d4393cf6f,c427432b2d3f387c
Lung_Opacity-1133.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1133.png,Lung_Opacity/masks/Lung_Opacity-1133.png,True,G03552,val,0e4e5d1966d7ddcaf51820460378aece,913957565a4b1b3a
Lung_Opacity-1134.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1134.png,Lung_Opacity/masks/Lung_Opacity-1134.png,True,G03553,test,260de84cc81b92575bc15b4dc7f7a98d,856a0a555b4b4f5d
Lung_Opacity-1135.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1135.png,Lung_Opacity/masks/Lung_Opacity-1135.png,True,G03554,test,e5a508a79f70ba9504aa4cb8024c9464,9169421e497f387b
Lung_Opacity-1136.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1136.png,Lung_Opacity/masks/Lung_Opacity-1136.png,True,G03555,val,9e6a9d5a4ccf768149919d9854b88049,d929d6125e14ee59
Lung_Opacity-1137.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1137.png,Lung_Opacity/masks/Lung_Opacity-1137.png,True,G03556,test,1a103ee17cd8f52a3f6654101b7c3f3c,dc10d21a3c3c3f3d
Lung_Opacity-1138.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1138.png,Lung_Opacity/masks/Lung_Opacity-1138.png,True,G03557,train,e5cf4be3291db757cd707fdfa113da21,913a045f1b5b5979
Lung_Opacity-1139.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1139.png,Lung_Opacity/masks/Lung_Opacity-1139.png,True,G03558,train,79dd4c21d73c51c5f9ecd2d9f95b212f,d27207722a61fb72
Lung_Opacity-114.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-114.png,Lung_Opacity/masks/Lung_Opacity-114.png,True,G03559,train,dc080b7579289ce9d5ab24b92eca796d,cd3f02134d0d6e4f
Lung_Opacity-1140.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1140.png,Lung_Opacity/masks/Lung_Opacity-1140.png,True,G03560,train,b3ec0ad957fb2ebb87dfef6ad4470e62,9d2a821d4d4c7e4f
Lung_Opacity-1141.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1141.png,Lung_Opacity/masks/Lung_Opacity-1141.png,True,G03561,train,ef0863599e0785e02d6418df8a54e46d,81781d1b5e5b585e
Lung_Opacity-1142.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1142.png,Lung_Opacity/masks/Lung_Opacity-1142.png,True,G03562,train,6de5ce8bb7e23b812eb2f0347176ecf5,dc0adb1d0ec25774
Lung_Opacity-1143.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1143.png,Lung_Opacity/masks/Lung_Opacity-1143.png,True,G03563,train,08f5e89e79b4f1e45cacb3b79b856158,c23c2c272d3f6527
Lung_Opacity-1144.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1144.png,Lung_Opacity/masks/Lung_Opacity-1144.png,True,G03564,train,ebb99f8f6c1b2075be055c0572be8356,d0d13e9e652dc662
Lung_Opacity-1145.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1145.png,Lung_Opacity/masks/Lung_Opacity-1145.png,True,G03565,val,a79063422a000118faeabe6003f37b3d,d13b58785d083e1f
Lung_Opacity-1146.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1146.png,Lung_Opacity/masks/Lung_Opacity-1146.png,True,G03566,train,eb8fc0f1f43ddab0e0504bdbf28635de,fd78079217471a5a
Lung_Opacity-1147.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1147.png,Lung_Opacity/masks/Lung_Opacity-1147.png,True,G03567,train,4f5bf6dbb0cd34149d9c89263e44132b,8b181d5e58686beb
Lung_Opacity-1148.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1148.png,Lung_Opacity/masks/Lung_Opacity-1148.png,True,G03568,val,eee5768e1c122aa87a9e48892c097579,88585f5e5a58696d
Lung_Opacity-1149.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1149.png,Lung_Opacity/masks/Lung_Opacity-1149.png,True,G03569,test,2832cc76e3e453e85c5c86bc309a6bf5,d42f45554c0a6e6f
Lung_Opacity-115.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-115.png,Lung_Opacity/masks/Lung_Opacity-115.png,True,G03570,train,674d2d3480a86edda0e323ffe51379d4,917fd610185f1a1f
Lung_Opacity-1150.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1150.png,Lung_Opacity/masks/Lung_Opacity-1150.png,True,G03571,train,46eb4d368ebe8ec73ef6000815ef0c55,995a1e796626a725
Lung_Opacity-1151.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1151.png,Lung_Opacity/masks/Lung_Opacity-1151.png,True,G03572,train,dfa61ce354efc2a210bb5121ca4de032,d5688f550f455e0b
Lung_Opacity-1152.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1152.png,Lung_Opacity/masks/Lung_Opacity-1152.png,True,G03573,test,e0b2e7561b680d53eaebc7916407de87,d43d031a5c4d1f5b
Lung_Opacity-1153.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1153.png,Lung_Opacity/masks/Lung_Opacity-1153.png,True,G03574,train,abe971b7b3c064a09c823619992609fb,d52a814dc6474777
Lung_Opacity-1154.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1154.png,Lung_Opacity/masks/Lung_Opacity-1154.png,True,G03575,train,bb4c8e6943a1d7368bdb2c0d868d2a05,943f03570f4c1e5e
Lung_Opacity-1155.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1155.png,Lung_Opacity/masks/Lung_Opacity-1155.png,True,G03576,val,2ae7fc5ad850bfff82e7892a57492e2c,d42d43d30b0ded2d
Lung_Opacity-1156.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1156.png,Lung_Opacity/masks/Lung_Opacity-1156.png,True,G03577,train,b575f465d8cb644ed528b708ce82d463,c627414b4fea55aa
Lung_Opacity-1157.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1157.png,Lung_Opacity/masks/Lung_Opacity-1157.png,True,G03578,train,91368d3a50b646265d521130c4e395f1,c42f470b6c6c6cea
Lung_Opacity-1158.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1158.png,Lung_Opacity/masks/Lung_Opacity-1158.png,True,G03579,train,5ae1399ab9ac912f1abf4bda444f0901,817b141a5b795b39
Lung_Opacity-1159.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1159.png,Lung_Opacity/masks/Lung_Opacity-1159.png,True,G03580,train,5465670d37692274e2fe05e5bd72f805,c02c9743cf1f1f0d
Lung_Opacity-116.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-116.png,Lung_Opacity/masks/Lung_Opacity-116.png,True,G03581,test,1d5b57fbc2493c9982cc5962fe3bfa17,81685f1b49cec6d6
Lung_Opacity-1160.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1160.png,Lung_Opacity/masks/Lung_Opacity-1160.png,True,G03582,train,3d987ac7017a0cf0490f15e963a5a4cc,c42d074b4c4e6f6e
Lung_Opacity-1161.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1161.png,Lung_Opacity/masks/Lung_Opacity-1161.png,True,G03583,train,a889dae652d50fae6aa610757b66a075,dd2a571b1f1c1e0c
Lung_Opacity-1162.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1162.png,Lung_Opacity/masks/Lung_Opacity-1162.png,True,G03584,train,0dae7f50b0c6d245685ec56af8d779a5,887f471e5849595e
Lung_Opacity-1163.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1163.png,Lung_Opacity/masks/Lung_Opacity-1163.png,True,G03585,test,b38b282c6407de784ab6e1a19e4aa54f,d42b414b2e6a2eed
Lung_Opacity-1164.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1164.png,Lung_Opacity/masks/Lung_Opacity-1164.png,True,G03586,train,5c3a416101695cfbf37ddcbbcd370b72,897814181b5f5f5e
Lung_Opacity-1165.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1165.png,Lung_Opacity/masks/Lung_Opacity-1165.png,True,G03587,train,72916cb74d355d50b380a4a9e121a538,d1381e1b185f395b
Lung_Opacity-1166.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1166.png,Lung_Opacity/masks/Lung_Opacity-1166.png,True,G03588,train,3473a0c3f841a7c8d94f06e23f508129,d42fc14b0f0f4aad
Lung_Opacity-1167.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1167.png,Lung_Opacity/masks/Lung_Opacity-1167.png,True,G03589,train,e65aa169318e02b8e249463209123b1f,d42f01570e0e2f6f
Lung_Opacity-1168.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1168.png,Lung_Opacity/masks/Lung_Opacity-1168.png,True,G03590,train,0ef51de132d1abe59a73766042015684,817a5e18185b7b3b
Lung_Opacity-1169.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1169.png,Lung_Opacity/masks/Lung_Opacity-1169.png,True,G03591,train,19856e482013fbb1a468589a8df6b049,c03d430d5e2b5a3f
Lung_Opacity-117.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-117.png,Lung_Opacity/masks/Lung_Opacity-117.png,True,G03592,val,d7e59ab12984ed86825e0bb11988bd9b,9f601b5b1b597344
Lung_Opacity-1170.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1170.png,Lung_Opacity/masks/Lung_Opacity-1170.png,True,G03593,val,d306c5cc74d90482c335323d0b5aef80,d52ea6f38a49f0d0
Lung_Opacity-1171.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1171.png,Lung_Opacity/masks/Lung_Opacity-1171.png,True,G03594,val,eaca97736e81867e8b914fce3394009b,c07cc86c6d6ccc97
Lung_Opacity-1172.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1172.png,Lung_Opacity/masks/Lung_Opacity-1172.png,True,G03595,test,6f5ee9ba8ae8f869525fae8ca310c81a,c02d535b58395a3f
Lung_Opacity-1173.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1173.png,Lung_Opacity/masks/Lung_Opacity-1173.png,True,G03596,test,b409ba2dcfad9a6bd394e29cb69c74a6,856c86095f6d2f47
Lung_Opacity-1174.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1174.png,Lung_Opacity/masks/Lung_Opacity-1174.png,True,G03597,val,9af8cfa5c9aad06b6af738726c19b566,c83dd21e5a595c5c
Lung_Opacity-1175.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1175.png,Lung_Opacity/masks/Lung_Opacity-1175.png,True,G03598,train,453b038b266811527e57f768e95f73e6,856e07475d6d6869
Lung_Opacity-1176.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1176.png,Lung_Opacity/masks/Lung_Opacity-1176.png,True,G03599,train,074fb76be97e4fc698222a74d3151621,bb7bc61851531b1a
Lung_Opacity-1177.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1177.png,Lung_Opacity/masks/Lung_Opacity-1177.png,True,G03600,train,67ca421a3384bd0d2c5d8de75cc29d46,81790cde1bf9ed80
Lung_Opacity-1178.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1178.png,Lung_Opacity/masks/Lung_Opacity-1178.png,True,G03601,train,34e96749cfb78d273a182a084d0f6140,d52b570c5f085b39
Lung_Opacity-1179.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1179.png,Lung_Opacity/masks/Lung_Opacity-1179.png,True,G03602,train,e596b2b3b0cbceffd35e910e8a522c26,d42d461a4d4d5c6f
Lung_Opacity-118.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-118.png,Lung_Opacity/masks/Lung_Opacity-118.png,True,G03603,train,5d8b879214bc3243f7bd737d9a23bfd4,947e01570b470f5f
Lung_Opacity-1180.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1180.png,Lung_Opacity/masks/Lung_Opacity-1180.png,True,G03604,val,cb8dce33ca03cfc04337bbd1eee2aa08,813f861a194d5f1f
Lung_Opacity-1181.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1181.png,Lung_Opacity/masks/Lung_Opacity-1181.png,True,G03605,train,c8b3b2f3929dedd7ec7cf1e3bf08e3d6,85685a6b3f26a6a6
Lung_Opacity-1182.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1182.png,Lung_Opacity/masks/Lung_Opacity-1182.png,True,G03606,train,02be748a328e7e285dc73f4c3a726e0b,996b451ccd1de02f
Lung_Opacity-1183.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1183.png,Lung_Opacity/masks/Lung_Opacity-1183.png,True,G03607,train,e610dfa3d9c1a24dad62fe374700451f,d82dc21b4c7d3c78
Lung_Opacity-1184.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1184.png,Lung_Opacity/masks/Lung_Opacity-1184.png,True,G03608,train,4c355085c383c9de046c1dab01514154,d42b43575d68685d
Lung_Opacity-1185.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1185.png,Lung_Opacity/masks/Lung_Opacity-1185.png,True,G03609,train,de1e64f87cdf2747859460977a891834,907bc6121d537b1b
Lung_Opacity-1186.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1186.png,Lung_Opacity/masks/Lung_Opacity-1186.png,True,G03610,train,6d48103cad5aa7604cdedef954bdd7f8,956a06595f2e1c6b
Lung_Opacity-1187.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1187.png,Lung_Opacity/masks/Lung_Opacity-1187.png,True,G03611,train,4b00aa99e7a01fee2505520a101f16f8,8118165e5f1a5f7a
Lung_Opacity-1188.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1188.png,Lung_Opacity/masks/Lung_Opacity-1188.png,True,G03612,train,0b1806231ad0549884930b96cdff7c31,dd7847c7b8488768
Lung_Opacity-1189.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1189.png,Lung_Opacity/masks/Lung_Opacity-1189.png,True,G03613,val,8e6fbccdf28016da9ff5fb03b1baf1c4,917a16121b7b1f1e
Lung_Opacity-119.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-119.png,Lung_Opacity/masks/Lung_Opacity-119.png,True,G03614,test,1fb72462c7ee5590f53d681fb4653ef2,d42f5d1d4e0f0e4a
Lung_Opacity-1190.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1190.png,Lung_Opacity/masks/Lung_Opacity-1190.png,True,G03615,test,95b9aae87c9960ab274906d406d71a3f,917fc6171a015d5d
Lung_Opacity-1191.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1191.png,Lung_Opacity/masks/Lung_Opacity-1191.png,True,G03616,test,da66661ab7539945fd368acf460418f0,d82f571a1d69083f
Lung_Opacity-1192.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1192.png,Lung_Opacity/masks/Lung_Opacity-1192.png,True,G03617,train,777438629eb03497881275b4d2748683,cd2c161b5b5d4e19
Lung_Opacity-1193.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1193.png,Lung_Opacity/masks/Lung_Opacity-1193.png,True,G03618,train,345f7432148def3a1bdd7d19b762a323,917bc5165e161e1e
Lung_Opacity-1194.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1194.png,Lung_Opacity/masks/Lung_Opacity-1194.png,True,G03619,train,fefe731aaa7701ecbe6b28dc188d6a8b,d22f0541474e6e6f
Lung_Opacity-1195.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1195.png,Lung_Opacity/masks/Lung_Opacity-1195.png,True,G03620,train,04a698a44af7cdd9a1639e46231913ab,946a070e0d5dbe5d
Lung_Opacity-1196.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1196.png,Lung_Opacity/masks/Lung_Opacity-1196.png,True,G03621,val,bc94ccb4dbbcfd11c51d852526a730e2,847ee2535969692d
Lung_Opacity-1197.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1197.png,Lung_Opacity/masks/Lung_Opacity-1197.png,True,G03622,train,85e429a300e4f4e2feab5bff534f4e4b,dc2ec34b0d1e6c2e
Lung_Opacity-1198.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1198.png,Lung_Opacity/masks/Lung_Opacity-1198.png,True,G03623,test,4cb73fc58cb9efcaf642dba8b89a132f,9938461a194ff9bc
Lung_Opacity-1199.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1199.png,Lung_Opacity/masks/Lung_Opacity-1199.png,True,G03624,train,19091e07ac33bee156a85321d6daa91b,d56e454b0e2e2e2d
Lung_Opacity-12.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-12.png,Lung_Opacity/masks/Lung_Opacity-12.png,True,G03625,train,e7cd25a7923e93efb13c9d656ade0894,813916465a5e7d5b
Lung_Opacity-120.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-120.png,Lung_Opacity/masks/Lung_Opacity-120.png,True,G03626,train,052a49caeb86f6e1f4d83e6f3068345c,ccefd35d42494e48
Lung_Opacity-1200.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1200.png,Lung_Opacity/masks/Lung_Opacity-1200.png,True,G03627,train,2a78c431631429fe47c0f6bd8d25bc9b,c94d161e1a5b4e5b
Lung_Opacity-1201.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1201.png,Lung_Opacity/masks/Lung_Opacity-1201.png,True,G03628,train,cf33cf120546258357aefa0dbb6386eb,c46e5b5969283d1d
Lung_Opacity-1202.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1202.png,Lung_Opacity/masks/Lung_Opacity-1202.png,True,G03629,train,42ade98c65ab66c9041c90518acb2ef7,856d87c70b4d0d5d
Lung_Opacity-1203.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1203.png,Lung_Opacity/masks/Lung_Opacity-1203.png,True,G03630,train,b40ee6e3879b10a83c5c072febf4a0bb,907fce1a19187d69
Lung_Opacity-1204.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1204.png,Lung_Opacity/masks/Lung_Opacity-1204.png,True,G03631,val,9ff7fbd6114273bcce427ecfa5ccf1db,c02ec75f1d4c1de8
Lung_Opacity-1205.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1205.png,Lung_Opacity/masks/Lung_Opacity-1205.png,True,G03632,train,7b8066ff143ff9fdd6c48c77cfb9161c,917e87121b0dbf19
Lung_Opacity-1206.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1206.png,Lung_Opacity/masks/Lung_Opacity-1206.png,True,G03633,train,44a844ae298b3864bb5f959cf0b05b80,d42ed3530f070f0f
Lung_Opacity-1207.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1207.png,Lung_Opacity/masks/Lung_Opacity-1207.png,True,G03634,val,ebe4d294c84f2341bc2874f2e2c216fc,d43fc31d1d42191f
Lung_Opacity-1208.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1208.png,Lung_Opacity/masks/Lung_Opacity-1208.png,True,G03635,train,100f992153f6c59f590ebdad229f58c7,d02d434a0f0f2f6f
Lung_Opacity-1209.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1209.png,Lung_Opacity/masks/Lung_Opacity-1209.png,True,G03636,test,c2f0e8683182dfa2bc37f261a4c87b62,9179ee841e0fb11e
Lung_Opacity-121.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-121.png,Lung_Opacity/masks/Lung_Opacity-121.png,True,G03637,test,bd84d938c23095ef385e2104a3205a3c,c07b3b55035b0a5f
Lung_Opacity-1210.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1210.png,Lung_Opacity/masks/Lung_Opacity-1210.png,True,G03638,train,84521079c9fc24e8de1f0fb47b382c87,993b960a191b5f1e
Lung_Opacity-1211.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1211.png,Lung_Opacity/masks/Lung_Opacity-1211.png,True,G03639,train,d9336eaa0fb49ae0d9b1099d9371c27f,817b04161c1f5f5f
Lung_Opacity-1212.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1212.png,Lung_Opacity/masks/Lung_Opacity-1212.png,True,G03640,train,0cb391cec07307d87f30d86e62048167,d42c42d74b0d3d6e
Lung_Opacity-1213.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1213.png,Lung_Opacity/masks/Lung_Opacity-1213.png,True,G03641,train,e9856f539481386c639f72cb2d1b18b8,81781f1f5a5a5a5a
Lung_Opacity-1214.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1214.png,Lung_Opacity/masks/Lung_Opacity-1214.png,True,G03642,train,b494cc5c34fd0e5d690fed54034d17f5,95780646195fce5b
Lung_Opacity-1215.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1215.png,Lung_Opacity/masks/Lung_Opacity-1215.png,True,G03643,train,3f7f1ba27aa51c8bc1e0b7134ca768c1,c42f1602531f5f7a
Lung_Opacity-1216.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1216.png,Lung_Opacity/masks/Lung_Opacity-1216.png,True,G03644,train,5930b8f6572beca4c9f86f0eff622f20,d07b57564a4a4a5b
Lung_Opacity-1217.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1217.png,Lung_Opacity/masks/Lung_Opacity-1217.png,True,G03645,train,aa775dedce6d612881b8095d3d2f6751,8938171a595f5c7a
Lung_Opacity-1218.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1218.png,Lung_Opacity/masks/Lung_Opacity-1218.png,True,G03646,train,46c25fb2152eb7b2342097d70e1592e1,817884593f5f394d
Lung_Opacity-1219.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1219.png,Lung_Opacity/masks/Lung_Opacity-1219.png,True,G03647,train,c09d815ef7a25b344ad2708453d2bcbd,803f061e5979787d
Lung_Opacity-122.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-122.png,Lung_Opacity/masks/Lung_Opacity-122.png,True,G03648,train,e350897776dab85f46e571da15d205d6,807d5443535b5d5b
Lung_Opacity-1220.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1220.png,Lung_Opacity/masks/Lung_Opacity-1220.png,True,G03649,train,c98a9e5d9a76ebb0e7c320094b498c83,8d688e5f8f143a2d
Lung_Opacity-1221.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1221.png,Lung_Opacity/masks/Lung_Opacity-1221.png,True,G03650,train,973519707d86beaad6306f7cf04dd972,817e041e1b5fdc5a
Lung_Opacity-1222.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1222.png,Lung_Opacity/masks/Lung_Opacity-1222.png,True,G03651,train,0d7c4f6ea4dd55a41e488a2f3a9979ee,c03f120b0b3f3d3d
Lung_Opacity-1223.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1223.png,Lung_Opacity/masks/Lung_Opacity-1223.png,True,G03652,val,a3b73d439e43ab0f81ff5be6bb2ef656,c03d430e2c2e1fbf
Lung_Opacity-1224.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1224.png,Lung_Opacity/masks/Lung_Opacity-1224.png,True,G03653,test,0cbcfff36ef2f38e1afd75c16a35bd4d,846c077b6173e333
Lung_Opacity-1225.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1225.png,Lung_Opacity/masks/Lung_Opacity-1225.png,True,G03654,val,3ea0ea2a39b35eb64422755eb6767b92,d23f01595f4a0a3f
Lung_Opacity-1226.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1226.png,Lung_Opacity/masks/Lung_Opacity-1226.png,True,G03655,val,a4556bec2bd63d54c208858433765830,dc2dd50b4f4c5c4c
Lung_Opacity-1227.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1227.png,Lung_Opacity/masks/Lung_Opacity-1227.png,True,G03656,test,f0c573c36ff262a84baaea7e6d93cc0a,813a4f5e581b7939
Lung_Opacity-1228.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1228.png,Lung_Opacity/masks/Lung_Opacity-1228.png,True,G03657,test,d14a31a6a3a09656f7cb49244ab2fb0c,90791f1a1b3a3af2
Lung_Opacity-1229.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1229.png,Lung_Opacity/masks/Lung_Opacity-1229.png,True,G03658,train,cfe96cb4fd33cf9955334749c90978cd,d43e65410f4f4e4e
Lung_Opacity-123.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-123.png,Lung_Opacity/masks/Lung_Opacity-123.png,True,G03659,test,b60a085415da4b4dad95ded957aa6c16,8971961e5b1e5a5a
Lung_Opacity-1230.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1230.png,Lung_Opacity/masks/Lung_Opacity-1230.png,True,G03660,test,f5360c6b3adbfb3bcbc3f422e2c26e0c,d969c61b154f1a4b
Lung_Opacity-1231.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1231.png,Lung_Opacity/masks/Lung_Opacity-1231.png,True,G03661,train,1eb50d0ef04104072aaeaf336ef2f267,d47f39b09a1a9a91
Lung_Opacity-1232.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1232.png,Lung_Opacity/masks/Lung_Opacity-1232.png,True,G03662,val,0bb418bd30364d7cb5441375546d2bf8,db7f841e181b1a5a
Lung_Opacity-1233.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1233.png,Lung_Opacity/masks/Lung_Opacity-1233.png,True,G03663,train,b508a980e95dd79e41858cab1adcf59f,897e07185c5d596b
Lung_Opacity-1234.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1234.png,Lung_Opacity/masks/Lung_Opacity-1234.png,True,G03664,train,c4058e7b6269bc76ef41cfc5247c4128,c46ebb515545574a
Lung_Opacity-1235.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1235.png,Lung_Opacity/masks/Lung_Opacity-1235.png,True,G03665,train,faf410254fab51bdca2e9beebd887b77,81785ec79e5aa949
Lung_Opacity-1236.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1236.png,Lung_Opacity/masks/Lung_Opacity-1236.png,True,G03666,train,4748e509fcba82afae79774a1daf463c,9f781e1b1b585bc0
Lung_Opacity-1237.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1237.png,Lung_Opacity/masks/Lung_Opacity-1237.png,True,G03667,train,2f8cc7c18aed13af52da4217dcbfa9b0,dc2f430a1a2b3dba
Lung_Opacity-1238.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1238.png,Lung_Opacity/masks/Lung_Opacity-1238.png,True,G03668,train,c6e41e8f6cf70e9f71c40d9997099d64,9669699878c96d87
Lung_Opacity-1239.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1239.png,Lung_Opacity/masks/Lung_Opacity-1239.png,True,G03669,train,c69f1ef466720f9c217e3ee05316f597,c42c52531f5b4f1d
Lung_Opacity-124.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-124.png,Lung_Opacity/masks/Lung_Opacity-124.png,True,G03670,train,fd34794b802c46b6ac6d4b635e7f1e77,d83d5b082d3cb1b5
Lung_Opacity-1240.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1240.png,Lung_Opacity/masks/Lung_Opacity-1240.png,True,G03671,train,dbbcc3bf33164407b831ad1b9213b122,d1281e1a1f595b5b
Lung_Opacity-1241.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1241.png,Lung_Opacity/masks/Lung_Opacity-1241.png,True,G03672,train,8b4ae6222bbe5c3fdd1587313e9e02c4,c129fa1859595f1e
Lung_Opacity-1242.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1242.png,Lung_Opacity/masks/Lung_Opacity-1242.png,True,G03673,train,e06ff7cb433d23ada1366acfe9b79900,917b461a175a3a3b
Lung_Opacity-1243.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1243.png,Lung_Opacity/masks/Lung_Opacity-1243.png,True,G03674,train,9c5d241aadf75286370ee30d1b2242c9,dd0a830f2cacadbc
Lung_Opacity-1244.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1244.png,Lung_Opacity/masks/Lung_Opacity-1244.png,True,G03675,val,7533b4281c770e3114a042c511bbdd32,c035ea424f4f7959
Lung_Opacity-1245.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1245.png,Lung_Opacity/masks/Lung_Opacity-1245.png,True,G03676,val,0ff23a637662f517eb128b9590cfb306,856847161a5b6f79
Lung_Opacity-1246.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1246.png,Lung_Opacity/masks/Lung_Opacity-1246.png,True,G03677,train,d145d565c4f78c245e183ea66af26491,9568870e5a59e579
Lung_Opacity-1247.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1247.png,Lung_Opacity/masks/Lung_Opacity-1247.png,True,G03678,test,bb16360e543d084f75b4017144eea064,c03f52101a1f7b3f
Lung_Opacity-1248.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1248.png,Lung_Opacity/masks/Lung_Opacity-1248.png,True,G03679,train,636511dafa83ee27dc6dda0e3c823169,857e47421b0c6f3d
Lung_Opacity-1249.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1249.png,Lung_Opacity/masks/Lung_Opacity-1249.png,True,G03680,train,78fc46b7a9dc3ad4803b5d7a7a01dbad,c13b46865a5f5a59
Lung_Opacity-125.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-125.png,Lung_Opacity/masks/Lung_Opacity-125.png,True,G03681,train,80c8da8227ed7ae97950dc3be9175591,c92d521b6d2cbda1
Lung_Opacity-1250.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1250.png,Lung_Opacity/masks/Lung_Opacity-1250.png,True,G03682,train,96b0a2ea1dbbf9fd54c764c966e61f5f,c54eea410b6dad2d
Lung_Opacity-1251.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1251.png,Lung_Opacity/masks/Lung_Opacity-1251.png,True,G03683,train,debf2422e0b3d7899ee48d0522d3f9bf,817a521b551716bf
Lung_Opacity-1252.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1252.png,Lung_Opacity/masks/Lung_Opacity-1252.png,True,G03684,train,a9f94a3d52463a3b9b34bbeae9f9a299,c13e299ec2e5163d
Lung_Opacity-1253.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1253.png,Lung_Opacity/masks/Lung_Opacity-1253.png,True,G03685,train,5af168d096d3cbbc4ec0ef387d13990b,95680e5c195d796b
Lung_Opacity-1254.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1254.png,Lung_Opacity/masks/Lung_Opacity-1254.png,True,G03686,train,19f19e355190cab7623d2c4febdb7ba6,dc2b431d1c1a1bbb
Lung_Opacity-1255.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1255.png,Lung_Opacity/masks/Lung_Opacity-1255.png,True,G03687,train,c301b1213f2ff5eaba9737d21a514a64,d11adbc20e3c3db1
Lung_Opacity-1256.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1256.png,Lung_Opacity/masks/Lung_Opacity-1256.png,True,G03688,train,a1f806e8f0b4d20d20ae9325c5521770,9b481e7a495d4e8e
Lung_Opacity-1257.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1257.png,Lung_Opacity/masks/Lung_Opacity-1257.png,True,G03689,train,360d4e68d43cbc99716943e3910f155b,cc3d53cb1d0e4d0e
Lung_Opacity-1258.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1258.png,Lung_Opacity/masks/Lung_Opacity-1258.png,True,G03690,train,e2a6f7acd50b3c9b650decb391e6688f,c03d36d23dc03fd1
Lung_Opacity-1259.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1259.png,Lung_Opacity/masks/Lung_Opacity-1259.png,True,G03691,val,c1dce79a400bc8bac8b175acbd093d58,9769555a387a2b29
Lung_Opacity-126.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-126.png,Lung_Opacity/masks/Lung_Opacity-126.png,True,G03692,train,caf00593bdda7f6a2c6def82cdd6b4e8,dc2c879a1d4b1a5d
Lung_Opacity-1260.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1260.png,Lung_Opacity/masks/Lung_Opacity-1260.png,True,G03693,val,c990a363ce7d0210620d1a86e7360a0f,803bc79e1e571e1a
Lung_Opacity-1261.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1261.png,Lung_Opacity/masks/Lung_Opacity-1261.png,True,G03694,val,90232276f5016b6050c92325fb78389e,957a581f5b3930a5
Lung_Opacity-1262.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1262.png,Lung_Opacity/masks/Lung_Opacity-1262.png,True,G03695,test,c45d0c5e247871fd3760334e2f242563,95287e5f6e60e485
Lung_Opacity-1263.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1263.png,Lung_Opacity/masks/Lung_Opacity-1263.png,True,G03696,train,6e06a845aa3c51dc6d9238944dacb401,946e43534b4b1e4f
Lung_Opacity-1264.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1264.png,Lung_Opacity/masks/Lung_Opacity-1264.png,True,G03697,test,9609dcf9e37d9ee1983fec9fa0821fbf,d43e0f05454f6e63
Lung_Opacity-1265.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1265.png,Lung_Opacity/masks/Lung_Opacity-1265.png,True,G03698,train,846d33ebbfe9e7c6ac0d630158dee077,9b6085145e1d5b7b
Lung_Opacity-1266.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1266.png,Lung_Opacity/masks/Lung_Opacity-1266.png,True,G03699,train,0539cb9df3c13d7156e5317b0aa88bb2,8a2bc245ab5c2e5f
Lung_Opacity-1267.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1267.png,Lung_Opacity/masks/Lung_Opacity-1267.png,True,G03700,test,fd5a5e4598873af92d6584ea49e83e9f,dd29564e4a4b5b4a
Lung_Opacity-1268.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1268.png,Lung_Opacity/masks/Lung_Opacity-1268.png,True,G03701,train,7a57f5a87286de36e7010aec6b67b7aa,887f06181b5b7979
Lung_Opacity-1269.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1269.png,Lung_Opacity/masks/Lung_Opacity-1269.png,True,G03702,train,cadae9aa1b85a01d45cfcf08005e8687,97601d5c181f787b
Lung_Opacity-127.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-127.png,Lung_Opacity/masks/Lung_Opacity-127.png,True,G03703,train,9333bb8b26bb0c1799b44ccd91900b82,cc3d5b0f4d0e2c3c
Lung_Opacity-1270.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1270.png,Lung_Opacity/masks/Lung_Opacity-1270.png,True,G03704,train,b7c541b85aad30378963158533461e71,991a1a5b6879e8f1
Lung_Opacity-1271.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1271.png,Lung_Opacity/masks/Lung_Opacity-1271.png,True,G03705,train,bc37908a9f019a0d416f7471a24c2209,9939c6c658471f5a
Lung_Opacity-1272.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1272.png,Lung_Opacity/masks/Lung_Opacity-1272.png,True,G03706,train,8d9a7571215980091d4993b5208d5e95,c42a094b5d5e6e6b
Lung_Opacity-1273.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1273.png,Lung_Opacity/masks/Lung_Opacity-1273.png,True,G03707,train,ce2b9a01b014f4ada999d190a41c076c,917a06161e1f387f
Lung_Opacity-1274.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1274.png,Lung_Opacity/masks/Lung_Opacity-1274.png,True,G03708,train,ea44ccfd250d70023f90dca1c2471274,816a50171b5b7b4f
Lung_Opacity-1275.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1275.png,Lung_Opacity/masks/Lung_Opacity-1275.png,True,G03709,val,abf30aaf00d289026d9dffc552282c67,c979125b491d3f29
Lung_Opacity-1276.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1276.png,Lung_Opacity/masks/Lung_Opacity-1276.png,True,G03710,train,0c0fe0e772c8453fe06ff751f126ece9,d42c07086e6de7f4
Lung_Opacity-1277.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1277.png,Lung_Opacity/masks/Lung_Opacity-1277.png,True,G03711,test,f61925f613b89011f4f812135236a9f1,c52c965b174a4f1d
Lung_Opacity-1278.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1278.png,Lung_Opacity/masks/Lung_Opacity-1278.png,True,G03712,train,14854a1666f6e8127b784dda0556571c,817a56551f1e591b
Lung_Opacity-1279.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1279.png,Lung_Opacity/masks/Lung_Opacity-1279.png,True,G03713,val,eed76b7dcb857cbe2f946d31e768aa41,c1395e18091d3f3f
Lung_Opacity-128.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-128.png,Lung_Opacity/masks/Lung_Opacity-128.png,True,G03714,train,c2f116ef8562deb7686f04f2008e76a4,d13a961e183b392f
Lung_Opacity-1280.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1280.png,Lung_Opacity/masks/Lung_Opacity-1280.png,True,G03715,val,09757cecc2acd310a39ed9c65c8a0aa2,c03f46074f4e4be9
Lung_Opacity-1281.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1281.png,Lung_Opacity/masks/Lung_Opacity-1281.png,True,G03716,test,6094b8d8822d90cd06f5358a5679851f,d17a5e161a1f191d
Lung_Opacity-1282.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1282.png,Lung_Opacity/masks/Lung_Opacity-1282.png,True,G03717,train,5ffa1596d6d1b903657470d80b99bfa6,802f43d71e48edbc
Lung_Opacity-1283.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1283.png,Lung_Opacity/masks/Lung_Opacity-1283.png,True,G03718,val,1824bc2f149de68880c25a38911a6ff4,dd2c43531f4d690d
Lung_Opacity-1284.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1284.png,Lung_Opacity/masks/Lung_Opacity-1284.png,True,G03719,test,bbbb98cb5f538084c3feee7f73b95be8,cd2e164f28fee0e0
Lung_Opacity-1285.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1285.png,Lung_Opacity/masks/Lung_Opacity-1285.png,True,G03720,test,149d51a33fcc83f7eaf9498806b42c79,d1295d1b3b3a3b0a
Lung_Opacity-1286.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1286.png,Lung_Opacity/masks/Lung_Opacity-1286.png,True,G03721,train,2f2d040b0b5f4f81e11d85283b368a7f,982e471b144f3d4f
Lung_Opacity-1287.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1287.png,Lung_Opacity/masks/Lung_Opacity-1287.png,True,G03722,train,97e9d84cd892a36e47ec73392cc1ebc6,9878e2471755171f
Lung_Opacity-1288.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1288.png,Lung_Opacity/masks/Lung_Opacity-1288.png,True,G03723,train,d00ee382959802993cef0b30f85e1453,c87f1556521d7c58
Lung_Opacity-1289.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1289.png,Lung_Opacity/masks/Lung_Opacity-1289.png,True,G03724,train,37385ed20c4628da8cfb6d00c18cfcf6,887d07461e587d79
Lung_Opacity-129.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-129.png,Lung_Opacity/masks/Lung_Opacity-129.png,True,G03725,train,a2228013ebe565f6771942613e0b9ab7,81781ede59f8e8e0
Lung_Opacity-1290.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1290.png,Lung_Opacity/masks/Lung_Opacity-1290.png,True,G03726,train,d70b1905c13046bf1a76f0898cda7106,8d4fc30b4b4b5a5a
Lung_Opacity-1291.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1291.png,Lung_Opacity/masks/Lung_Opacity-1291.png,True,G03727,train,46668c8034051a4bd045b9747c33ed0a,933a581f39587353
Lung_Opacity-1292.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1292.png,Lung_Opacity/masks/Lung_Opacity-1292.png,True,G03728,train,12193486c55e2494f5a82349063b00a0,d012163c39adadbd
Lung_Opacity-1293.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1293.png,Lung_Opacity/masks/Lung_Opacity-1293.png,True,G03729,train,44a265e0ecb00d5f78362be8f4e344f0,c53e1643571e6c5a
Lung_Opacity-1294.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1294.png,Lung_Opacity/masks/Lung_Opacity-1294.png,True,G03730,train,0ab6acb68c242ccb2ab4fef14274f636,da2d414f494f5c5c
Lung_Opacity-1295.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1295.png,Lung_Opacity/masks/Lung_Opacity-1295.png,True,G03731,test,5eedbf8d47ccd0511cdff1486aa6ae21,916acf3e1e491d49
Lung_Opacity-1296.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1296.png,Lung_Opacity/masks/Lung_Opacity-1296.png,True,G03732,train,b2da51376d1564b913eebc399a17ed08,807bc41c185b7b7b
Lung_Opacity-1297.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1297.png,Lung_Opacity/masks/Lung_Opacity-1297.png,True,G03733,train,d1ebfecaeb155dcf588a1ea2a875cefa,857e97580f5c3c2c
Lung_Opacity-1298.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1298.png,Lung_Opacity/masks/Lung_Opacity-1298.png,True,G03734,train,47d996ab8e6689eda2adf6c01f9a190b,987bc71c103b3da5
Lung_Opacity-1299.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1299.png,Lung_Opacity/masks/Lung_Opacity-1299.png,True,G03735,train,75d4b35c0f48d4db9be2bbfb46633fad,9a6886fff0e6a411
Lung_Opacity-13.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-13.png,Lung_Opacity/masks/Lung_Opacity-13.png,True,G03736,val,db697f31e97a6919cc04baf365bf2947,c42dc7475d086d6e
Lung_Opacity-130.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-130.png,Lung_Opacity/masks/Lung_Opacity-130.png,True,G03737,val,23493e0c4ace773dcdef35bb2e96ddc6,d56cc6d21a4b874b
Lung_Opacity-1300.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1300.png,Lung_Opacity/masks/Lung_Opacity-1300.png,True,G03738,test,7213aa3099a2f850383ce885057c3532,8d2f061b091f0f5f
Lung_Opacity-1301.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1301.png,Lung_Opacity/masks/Lung_Opacity-1301.png,True,G03739,train,452ff682f2e784290cfb8aae547fa58a,d17f57161657061a
Lung_Opacity-1302.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1302.png,Lung_Opacity/masks/Lung_Opacity-1302.png,True,G03740,train,39d7e4453b3df6ff045ebe04e278478a,8d67070f6d657458
Lung_Opacity-1303.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1303.png,Lung_Opacity/masks/Lung_Opacity-1303.png,True,G03741,train,da8b7523181e3afb24275866bdf22ab5,9334d57e70736622
Lung_Opacity-1304.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1304.png,Lung_Opacity/masks/Lung_Opacity-1304.png,True,G03742,train,102eda52ad236878a5e98fe76b803149,88680a1f595d1edf
Lung_Opacity-1305.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1305.png,Lung_Opacity/masks/Lung_Opacity-1305.png,True,G03743,train,8d5102448713715bd52657c4806396c5,d03d46570b5f2a4d
Lung_Opacity-1306.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1306.png,Lung_Opacity/masks/Lung_Opacity-1306.png,True,G03744,train,b5abc451b22c1fbedd2b4788ee94a99e,812d2a3d555d696e
Lung_Opacity-1307.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1307.png,Lung_Opacity/masks/Lung_Opacity-1307.png,True,G03745,train,8711fdb007a2dd90294346809d3dfb05,c92a169a5b5a4d5d
Lung_Opacity-1308.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1308.png,Lung_Opacity/masks/Lung_Opacity-1308.png,True,G03746,train,927a5831904beaa4735932601c9a6c6f,c9ffd61c485c4859
Lung_Opacity-1309.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1309.png,Lung_Opacity/masks/Lung_Opacity-1309.png,True,G03747,train,66ded4d3b723dacb31865577a17e2ae3,cc4f4b4d0c2cadad
Lung_Opacity-131.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-131.png,Lung_Opacity/masks/Lung_Opacity-131.png,True,G03748,train,6756fbc3a635bfe640a4516be7576f15,913a125d385bcbda
Lung_Opacity-1310.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1310.png,Lung_Opacity/masks/Lung_Opacity-1310.png,True,G03749,train,85837b0b63fb86e73e5bbb3b47c305ca,987f461e1a69293d
Lung_Opacity-1311.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1311.png,Lung_Opacity/masks/Lung_Opacity-1311.png,True,G03750,train,cb47d52bc9f44d8e7927c85836597bb1,917f061a417a7763
Lung_Opacity-1312.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1312.png,Lung_Opacity/masks/Lung_Opacity-1312.png,True,G03751,train,0ca432c644cd605bc0f8da5688de0608,d63ed151d14b8f06
Lung_Opacity-1313.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1313.png,Lung_Opacity/masks/Lung_Opacity-1313.png,True,G03752,val,b372c79d25d1c70f85f78a4a3a48ff17,cd67431f49696ce0
Lung_Opacity-1314.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1314.png,Lung_Opacity/masks/Lung_Opacity-1314.png,True,G03753,val,786ea8b772b940603b155ac2e24588d7,d52e275d1d0d2c6c
Lung_Opacity-1315.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1315.png,Lung_Opacity/masks/Lung_Opacity-1315.png,True,G03754,val,bb44fb1ad6fcd685f5cf0060e3e9ea7e,917887141e1f1f5e
Lung_Opacity-1316.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1316.png,Lung_Opacity/masks/Lung_Opacity-1316.png,True,G03755,test,d9e418d73cf4a2a031f60191bccd6ad0,dc2dd15b1e4b810f
Lung_Opacity-1317.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1317.png,Lung_Opacity/masks/Lung_Opacity-1317.png,True,G03756,train,c4f76a812ad4a02ae5fe8599877bdf6b,894f865b6de0f0f0
Lung_Opacity-1318.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1318.png,Lung_Opacity/masks/Lung_Opacity-1318.png,True,G03757,test,4eb6e4d54d95d2ccbbda4ecdce5965e6,8967870e2d656575
Lung_Opacity-1319.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1319.png,Lung_Opacity/masks/Lung_Opacity-1319.png,True,G03758,test,bb6af16a7f505f7c07823cfc1390a2a7,d52a53535f4c0b4d
Lung_Opacity-132.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-132.png,Lung_Opacity/masks/Lung_Opacity-132.png,True,G03759,val,3fc48953a0a13866dd9c724994e18401,c422414c62fbfbb6
Lung_Opacity-1320.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1320.png,Lung_Opacity/masks/Lung_Opacity-1320.png,True,G03760,train,1fcd5e910a6033df2b55caec38b6ecec,897be51214c6cece
Lung_Opacity-1321.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1321.png,Lung_Opacity/masks/Lung_Opacity-1321.png,True,G03761,train,e029874ba424914444fc7cc9eb036cff,917fe61054193dd6
Lung_Opacity-1322.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1322.png,Lung_Opacity/masks/Lung_Opacity-1322.png,True,G03762,train,9886086bcc66c12f7a6bfece5a8e8877,997d45c61b184d6d
Lung_Opacity-1323.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1323.png,Lung_Opacity/masks/Lung_Opacity-1323.png,True,G03763,val,913fac8bbed6362f9129bc9916da6a52,947ec3d7031f181d
Lung_Opacity-1324.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1324.png,Lung_Opacity/masks/Lung_Opacity-1324.png,True,G03764,train,6e1bafc8bb475985cdddfd34c429e6b2,d0291f8e1e3c4d5b
Lung_Opacity-1325.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1325.png,Lung_Opacity/masks/Lung_Opacity-1325.png,True,G03765,train,a530e5789933fb2ac02deaf3224efe97,817f8c18191ff81f
Lung_Opacity-1326.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1326.png,Lung_Opacity/masks/Lung_Opacity-1326.png,True,G03766,train,5ada138fdbd315419c686847df9af460,903fe5018f5f1d0d
Lung_Opacity-1327.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1327.png,Lung_Opacity/masks/Lung_Opacity-1327.png,True,G03767,train,0eb98298afb079338da6af0e6043d085,cc3c971343c51f1d
Lung_Opacity-1328.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1328.png,Lung_Opacity/masks/Lung_Opacity-1328.png,True,G03768,train,d1f37e01cd91a1bdf9d4de44e4404a49,d56b85161657161f
Lung_Opacity-1329.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1329.png,Lung_Opacity/masks/Lung_Opacity-1329.png,True,G03769,train,a4ff0f5f817a765c92b8f631b2e674b8,c538ae0f9b08bc6d
Lung_Opacity-133.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-133.png,Lung_Opacity/masks/Lung_Opacity-133.png,True,G03770,train,c1ee9106852a3ef1780f3ea183e625fd,d835531ea43e36b4
Lung_Opacity-1330.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1330.png,Lung_Opacity/masks/Lung_Opacity-1330.png,True,G03771,train,d6a76886376d51c33b5ebea256c1dc9d,d02d5f525b484d6e
Lung_Opacity-1331.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1331.png,Lung_Opacity/masks/Lung_Opacity-1331.png,True,G03772,train,dcf18f9d5697bbc21010659069983272,c8bdc796159e1ec0
Lung_Opacity-1332.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1332.png,Lung_Opacity/masks/Lung_Opacity-1332.png,True,G03773,train,d31e1cf79af33af00f979e3f4f9ea4f9,886f86f9f1f0d0d0
Lung_Opacity-1333.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1333.png,Lung_Opacity/masks/Lung_Opacity-1333.png,True,G03774,test,5d2a3777a1029e773ef89fc8cc9e74fc,d03953475e1e2d2b
Lung_Opacity-1334.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1334.png,Lung_Opacity/masks/Lung_Opacity-1334.png,True,G03775,val,d1345a1539eb8335900bc571460f4065,c6250b4b6c2e3ee5
Lung_Opacity-1335.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1335.png,Lung_Opacity/masks/Lung_Opacity-1335.png,True,G03776,test,1ee079bb6be25da995f02083959a60b8,987dc682595d6d68
Lung_Opacity-1336.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1336.png,Lung_Opacity/masks/Lung_Opacity-1336.png,True,G03777,test,957094726d93238d8dc0a0f30a9b111f,9168c51f1b5f3236
Lung_Opacity-1337.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1337.png,Lung_Opacity/masks/Lung_Opacity-1337.png,True,G03778,train,b5000aca6c2bff04cdd07d6ba5f3230b,c42d16c3c74f1e3c
Lung_Opacity-1338.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1338.png,Lung_Opacity/masks/Lung_Opacity-1338.png,True,G03779,train,3678cb6f0ed91a548d4f3d100a0df272,c43c3fc331b2f423
Lung_Opacity-1339.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1339.png,Lung_Opacity/masks/Lung_Opacity-1339.png,True,G03780,train,5a79336e895a3bc36aaecb98490cb96e,947f035f0f1c1e0e
Lung_Opacity-134.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-134.png,Lung_Opacity/masks/Lung_Opacity-134.png,True,G03781,train,591f17893764df752e8527c4e48e52a3,903dd3da5ac21e6a
Lung_Opacity-1340.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1340.png,Lung_Opacity/masks/Lung_Opacity-1340.png,True,G03782,train,90cd93542d8593efb650b41f1b8574e6,91691ece5a3929e9
Lung_Opacity-1341.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1341.png,Lung_Opacity/masks/Lung_Opacity-1341.png,True,G03783,train,7839fa1de527692b7e014bd1562349f9,d93992121f3d3d31
Lung_Opacity-1342.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1342.png,Lung_Opacity/masks/Lung_Opacity-1342.png,True,G03784,train,d22c6a2f5f4214ddb27f9c2e046416b6,942c478b1c6f6c7c
Lung_Opacity-1343.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1343.png,Lung_Opacity/masks/Lung_Opacity-1343.png,True,G03785,train,52cb3394840f0205647d04ed7dbce68c,890d9e5aa3a3a7a5
Lung_Opacity-1344.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1344.png,Lung_Opacity/masks/Lung_Opacity-1344.png,True,G03786,train,44f5d3bc3f6d5bb3db2ce8a58d24a875,c06d165f5b1e6878
Lung_Opacity-1345.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1345.png,Lung_Opacity/masks/Lung_Opacity-1345.png,True,G03787,train,9ca60b032d0ebdfd7ba4eeac211015cd,9128041e5e5b5b7f
Lung_Opacity-1346.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1346.png,Lung_Opacity/masks/Lung_Opacity-1346.png,True,G03788,train,3dfb782ea468c488744c33d6637e33ac,dc2d49170a1f1b9b
Lung_Opacity-1347.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1347.png,Lung_Opacity/masks/Lung_Opacity-1347.png,True,G03789,train,ea0175a8202666f26546b7946f587cd2,d62a01452e2b7b7b
Lung_Opacity-1348.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1348.png,Lung_Opacity/masks/Lung_Opacity-1348.png,True,G03790,train,4e9de4aa77563545eb0c912d0a01c8fe,857832975ad46a9b
Lung_Opacity-1349.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1349.png,Lung_Opacity/masks/Lung_Opacity-1349.png,True,G03791,train,b9faaa1a5671e0b3302565181a47e783,8979e796869438cd
Lung_Opacity-135.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-135.png,Lung_Opacity/masks/Lung_Opacity-135.png,True,G03792,val,dc0450050ab8d5fc21d5bc16fd154085,81780e1f1f597879
Lung_Opacity-1350.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1350.png,Lung_Opacity/masks/Lung_Opacity-1350.png,True,G03793,val,bdc73c94ef812cfa91ecc6b8c57f74d7,d12a571e181b3a3f
Lung_Opacity-1351.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1351.png,Lung_Opacity/masks/Lung_Opacity-1351.png,True,G03794,test,7eea0c0c77ce27a6ddf653d3adbc994e,89780e5e59795969
Lung_Opacity-1352.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1352.png,Lung_Opacity/masks/Lung_Opacity-1352.png,True,G03795,test,2cf4f4fd91fefa61cb8c7562ee4ca95a,903a531edc191e7b
Lung_Opacity-1353.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1353.png,Lung_Opacity/masks/Lung_Opacity-1353.png,True,G03796,val,98d982f9b57454f39bdef179bc92d043,91291f1e1b3b3b29
Lung_Opacity-1354.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1354.png,Lung_Opacity/masks/Lung_Opacity-1354.png,True,G03797,test,2fc1eed8bf94c39773afdf88dd49f03e,d43f430b4e0d2d3d
Lung_Opacity-1355.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1355.png,Lung_Opacity/masks/Lung_Opacity-1355.png,True,G03798,train,27c5de52c4824168926c0d8e2bc50239,d4bd2fc20ac16bc3
Lung_Opacity-1356.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1356.png,Lung_Opacity/masks/Lung_Opacity-1356.png,True,G03799,train,360fbbb701d60f2498368c181895a7a7,897b47461e59495d
Lung_Opacity-1357.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1357.png,Lung_Opacity/masks/Lung_Opacity-1357.png,True,G03800,train,e59921e113d50dcea774f616fc89e694,d93b471b1a1d4539
Lung_Opacity-1358.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1358.png,Lung_Opacity/masks/Lung_Opacity-1358.png,True,G03801,train,553d00ab1dbdd4f7b76ac3d0fb8a8324,985b471839b9a6e6
Lung_Opacity-1359.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1359.png,Lung_Opacity/masks/Lung_Opacity-1359.png,True,G03802,val,9b3b1dcac4541baff2d5e78b4db26cff,cc0f0b6db8b1b193
Lung_Opacity-136.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-136.png,Lung_Opacity/masks/Lung_Opacity-136.png,True,G03803,train,524dff86b2a918359c742024b305e733,817d405758515f5f
Lung_Opacity-1360.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1360.png,Lung_Opacity/masks/Lung_Opacity-1360.png,True,G03804,train,42d552db141729f92a1dc5fb0791b4b3,956e054f485f0a5f
Lung_Opacity-1361.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1361.png,Lung_Opacity/masks/Lung_Opacity-1361.png,True,G03805,train,d9e18197109684788f8138e7a3029e27,c13b52db5d595119
Lung_Opacity-1362.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1362.png,Lung_Opacity/masks/Lung_Opacity-1362.png,True,G03806,train,e8109ddc4e7c52cae26e046537910555,9463684f3947196f
Lung_Opacity-1363.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1363.png,Lung_Opacity/masks/Lung_Opacity-1363.png,True,G03807,test,af449b78b19c2fa1e997bc8894532e69,d63d2151bb49654e
Lung_Opacity-1364.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1364.png,Lung_Opacity/masks/Lung_Opacity-1364.png,True,G03808,train,e01e27aae9f869e8f6b31d62d8656690,c43e66c30b3d2b4d
Lung_Opacity-1365.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1365.png,Lung_Opacity/masks/Lung_Opacity-1365.png,True,G03809,train,460b9c801f0cc62b3c3072ed2758f090,91194f4e585c7a79
Lung_Opacity-1366.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1366.png,Lung_Opacity/masks/Lung_Opacity-1366.png,True,G03810,train,6682ce1131e5c4cf79f6b3b39130f08d,993f441c055f1d4f
Lung_Opacity-1367.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1367.png,Lung_Opacity/masks/Lung_Opacity-1367.png,True,G03811,train,cd5768997683d051d366fd28a2fca474,9178060e1979e97f
Lung_Opacity-1368.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1368.png,Lung_Opacity/masks/Lung_Opacity-1368.png,True,G03812,train,49e7d660b7eccfa985bc619f8953e7bf,d23d2f0f0e2e6869
Lung_Opacity-1369.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1369.png,Lung_Opacity/masks/Lung_Opacity-1369.png,True,G03813,train,f17f8af2755a456842453ed33ada2719,c43c97430b3e2e3d
Lung_Opacity-137.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-137.png,Lung_Opacity/masks/Lung_Opacity-137.png,True,G03814,val,e57cee36badc49b17262cf710d0bba41,9770c185959d9d95
Lung_Opacity-1370.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1370.png,Lung_Opacity/masks/Lung_Opacity-1370.png,True,G03815,val,fd0d7aab6e3ed3edb529418c95ff5dff,c46fd15a0e4b494f
Lung_Opacity-1371.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1371.png,Lung_Opacity/masks/Lung_Opacity-1371.png,True,G03816,test,f5420ff54164e5d1c025eb9b4aba119f,c42d024f3c2d37b7
Lung_Opacity-1372.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1372.png,Lung_Opacity/masks/Lung_Opacity-1372.png,True,G03817,test,bba9e5543fbb070ee118605d5710fbb5,c03d53cb0b0b3f1b
Lung_Opacity-1373.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1373.png,Lung_Opacity/masks/Lung_Opacity-1373.png,True,G03818,train,9f2b974228e5ef1fe27ea7ac242d0e74,db3b86161a2b3b38
Lung_Opacity-1374.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1374.png,Lung_Opacity/masks/Lung_Opacity-1374.png,True,G03819,train,ed9fdbb35e29197c10750638383c389b,903e1e5a5858f8fa
Lung_Opacity-1375.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1375.png,Lung_Opacity/masks/Lung_Opacity-1375.png,True,G03820,train,68dd13fb8e6908b6c893fc8f4a7fed1f,8168165e5b797a69
Lung_Opacity-1376.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1376.png,Lung_Opacity/masks/Lung_Opacity-1376.png,True,G03821,train,a316a30a8b35f105b9b334f74c9eae24,c13d96425a3ce8eb
Lung_Opacity-1377.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1377.png,Lung_Opacity/masks/Lung_Opacity-1377.png,True,G03822,train,81a1302413971896ec9afe2d21a0494b,cc2d6355fe484d54
Lung_Opacity-1378.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1378.png,Lung_Opacity/masks/Lung_Opacity-1378.png,True,G03823,train,bf5ae0b7a55b8ec8a9ce7c922ca4b9be,d13b53925a5b5a19
Lung_Opacity-1379.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1379.png,Lung_Opacity/masks/Lung_Opacity-1379.png,True,G03824,train,55497597870ff12cbfa28e8b7869c3c5,957e0f441c6c6cbc
Lung_Opacity-138.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-138.png,Lung_Opacity/masks/Lung_Opacity-138.png,True,G03825,train,162c9181daf5b2f5eb03fc33608e75ff,907bd4105a5a7b3b
Lung_Opacity-1380.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1380.png,Lung_Opacity/masks/Lung_Opacity-1380.png,True,G03826,train,5a9be9be84df14a0b75e2656e2101667,c87f07021f1f5a1d
Lung_Opacity-1381.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1381.png,Lung_Opacity/masks/Lung_Opacity-1381.png,True,G03827,train,434a2f941875999522932212fc965af1,807e9c1c1e1e1e5f
Lung_Opacity-1382.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1382.png,Lung_Opacity/masks/Lung_Opacity-1382.png,True,G03828,train,40b0b15faea0b8eb6eedaa28a96f54ea,dd3d04426bc7c743
Lung_Opacity-1383.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1383.png,Lung_Opacity/masks/Lung_Opacity-1383.png,True,G03829,train,b56b6180d542c1fd1a565d3c96597d6c,842f078f1d1d0f5d
Lung_Opacity-1384.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1384.png,Lung_Opacity/masks/Lung_Opacity-1384.png,True,G03830,train,93484e2a9a4636238724e5498e52faae,c12f160f1e787c69
Lung_Opacity-1385.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1385.png,Lung_Opacity/masks/Lung_Opacity-1385.png,True,G03831,train,a9580d9e2e8f2f0064f669feaa61fbd7,dc3d470b0b0f6c6c
Lung_Opacity-1386.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1386.png,Lung_Opacity/masks/Lung_Opacity-1386.png,True,G03832,val,32f119d47dfd8860149b177e0737a557,c42e5f0b4b0f0f6c
Lung_Opacity-1387.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1387.png,Lung_Opacity/masks/Lung_Opacity-1387.png,True,G03833,val,78568fb54f0ec31ad9494985a7976c28,996c0e5e5968e9e1
Lung_Opacity-1388.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1388.png,Lung_Opacity/masks/Lung_Opacity-1388.png,True,G03834,val,9f7ab12e027749fd2efd5b5ccf0f53d7,983b4b1e18497df2
Lung_Opacity-1389.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1389.png,Lung_Opacity/masks/Lung_Opacity-1389.png,True,G03835,test,8170a33ffb4acb796af7daafcdb0fa09,852c075e5f7e6858
Lung_Opacity-139.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-139.png,Lung_Opacity/masks/Lung_Opacity-139.png,True,G03836,test,cd98116c60d39009d4b8fefc3ea632b1,d56e0c4f0f0d4e4e
Lung_Opacity-1390.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1390.png,Lung_Opacity/masks/Lung_Opacity-1390.png,True,G03837,train,ac1f0e648d7a0cfa55ae8d9e93b5344a,cd7a17031e1c4b5b
Lung_Opacity-1391.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1391.png,Lung_Opacity/masks/Lung_Opacity-1391.png,True,G03838,test,3d4dd413e2c6b3aa04b5ffe7d594959c,837a800515373f7f
Lung_Opacity-1392.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1392.png,Lung_Opacity/masks/Lung_Opacity-1392.png,True,G03839,train,6262790f32352eb9effbec891767526c,802d5b1b5f197c5c
Lung_Opacity-1393.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1393.png,Lung_Opacity/masks/Lung_Opacity-1393.png,True,G03840,train,f2312c7180321dbd0df604c0f2825fc0,c42f07130d1f1f6d
Lung_Opacity-1394.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1394.png,Lung_Opacity/masks/Lung_Opacity-1394.png,True,G03841,train,dc322b348f4a1cc20a16058f1638de12,d53b7ae1c3602cb2
Lung_Opacity-1395.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1395.png,Lung_Opacity/masks/Lung_Opacity-1395.png,True,G03842,train,d0eac658c372b5c352e9ca5a88b06ecd,dd2d361213776313
Lung_Opacity-1396.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1396.png,Lung_Opacity/masks/Lung_Opacity-1396.png,True,G03843,train,0ee7d534a7c7e6d356c159164d6de04f,cd7c8743174d1a4d
Lung_Opacity-1397.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1397.png,Lung_Opacity/masks/Lung_Opacity-1397.png,True,G03844,train,a92fb39439bb15f3346a72a1764d9bb3,817b550486d79e5b
Lung_Opacity-1398.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1398.png,Lung_Opacity/masks/Lung_Opacity-1398.png,True,G03845,val,7bb56489aaeb20ce6f06f334eb17d8d4,cc2c965a1d6c1b6d
Lung_Opacity-1399.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1399.png,Lung_Opacity/masks/Lung_Opacity-1399.png,True,G03846,test,458ae34720b27d8c89a6ed2e1827d677,d12b56161e1b5b1e
Lung_Opacity-14.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-14.png,Lung_Opacity/masks/Lung_Opacity-14.png,True,G03847,train,6aa2426124b75f1a7890a38521658498,d43e49092fc92fc9
Lung_Opacity-140.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-140.png,Lung_Opacity/masks/Lung_Opacity-140.png,True,G03848,train,99b91f8e3fdb8f5fcd43ddcdadf47aa2,8979c60e39392d3d
Lung_Opacity-1400.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1400.png,Lung_Opacity/masks/Lung_Opacity-1400.png,True,G03849,train,2fb0fcd7fa377e0770fd900d58cb4860,c13bd60a383b3d3a
Lung_Opacity-1401.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1401.png,Lung_Opacity/masks/Lung_Opacity-1401.png,True,G03850,val,5c22092656db1a0b6193a27a78f46245,d03b5e08083dafaf
Lung_Opacity-1402.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1402.png,Lung_Opacity/masks/Lung_Opacity-1402.png,True,G03851,val,999bc9a6d940b09007d87ccfeafa0514,817a161a1b5b5d79
Lung_Opacity-1403.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1403.png,Lung_Opacity/masks/Lung_Opacity-1403.png,True,G03852,train,c683d7e7bc0ff3e0aceadceaf8b5ffb5,807f571a5b696c64
Lung_Opacity-1404.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1404.png,Lung_Opacity/masks/Lung_Opacity-1404.png,True,G03853,train,78be3d4d572c95784b187ddc52634fc0,817a57145a6979d9
Lung_Opacity-1405.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1405.png,Lung_Opacity/masks/Lung_Opacity-1405.png,True,G03854,train,71a3365a1ce4fc72f5c53f0bf7ef90a3,dc2e425b2d6c6a6a
Lung_Opacity-1406.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1406.png,Lung_Opacity/masks/Lung_Opacity-1406.png,True,G03855,train,7b5386fe52a975735a9670c515584628,853e001d5d4c5f5f
Lung_Opacity-1407.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1407.png,Lung_Opacity/masks/Lung_Opacity-1407.png,True,G03856,val,87be06b293826fdd8d56fbeb033c6fcd,81795e86ca5bb92c
Lung_Opacity-1408.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1408.png,Lung_Opacity/masks/Lung_Opacity-1408.png,True,G03857,test,9030976750186c2f02c0f94966a40f07,dd2cf24b1f680d4c
Lung_Opacity-1409.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1409.png,Lung_Opacity/masks/Lung_Opacity-1409.png,True,G03858,test,00d22b5bcd570be57b9d5bdc2499e3b0,cd3a524f0e2d1d2d
Lung_Opacity-141.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-141.png,Lung_Opacity/masks/Lung_Opacity-141.png,True,G03859,train,9225d65eea065d9a2021373d1294dc02,d03dd7935e185a1a
Lung_Opacity-1410.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1410.png,Lung_Opacity/masks/Lung_Opacity-1410.png,True,G03860,train,f271204e71baf82eba7f81374585ba93,903ff65358405d5b
Lung_Opacity-1411.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1411.png,Lung_Opacity/masks/Lung_Opacity-1411.png,True,G03861,train,a06917c8d0377a35d2f0a091896f306e,d42cc3cb4f6d480f
Lung_Opacity-1412.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1412.png,Lung_Opacity/masks/Lung_Opacity-1412.png,True,G03862,train,87133886244ee316b5a5402915b2ce4d,da1d5a2f25257570
Lung_Opacity-1413.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1413.png,Lung_Opacity/masks/Lung_Opacity-1413.png,True,G03863,train,28ef13505b4beda1d7fa9f763cedce6d,9129a6861e571f79
Lung_Opacity-1414.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1414.png,Lung_Opacity/masks/Lung_Opacity-1414.png,True,G03864,train,79fed740ccdaa1d4416dcfb286a7ea0f,987bc71e18594979
Lung_Opacity-1415.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1415.png,Lung_Opacity/masks/Lung_Opacity-1415.png,True,G03865,train,e6511ff5f8462622eb46ad928a8e9e91,956c461a1b4f1f4e
Lung_Opacity-1416.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1416.png,Lung_Opacity/masks/Lung_Opacity-1416.png,True,G03866,train,5a6897f744ee96e6c3ba9ae79c9fc790,cd2e12534f175b49
Lung_Opacity-1417.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1417.png,Lung_Opacity/masks/Lung_Opacity-1417.png,True,G03867,train,60447dc4c6901c745fba479530afdb94,d62d484b6b262e3e
Lung_Opacity-1418.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1418.png,Lung_Opacity/masks/Lung_Opacity-1418.png,True,G03868,train,805733b80f102b89df44d0b110d220a3,9f7a1e1f105a5a58
Lung_Opacity-1419.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1419.png,Lung_Opacity/masks/Lung_Opacity-1419.png,True,G03869,train,3364f52759d23deca77ed596c8ff5bc3,c02ec75f1e4a59cc
Lung_Opacity-142.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-142.png,Lung_Opacity/masks/Lung_Opacity-142.png,True,G03870,train,fe7770f479cb3a3d33b5e72f57f18d20,9f601c1f1a787a53
Lung_Opacity-1420.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1420.png,Lung_Opacity/masks/Lung_Opacity-1420.png,True,G03871,train,b54ae7e03dda747fdf2e885a241c685d,cc2e834d0ce8c7f3
Lung_Opacity-1421.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1421.png,Lung_Opacity/masks/Lung_Opacity-1421.png,True,G03872,val,513188b2e62a08968640547d0375ac45,c83c8a8e8eb0afb6
Lung_Opacity-1422.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1422.png,Lung_Opacity/masks/Lung_Opacity-1422.png,True,G03873,train,8b6b18226c604d358990926d9b5a4382,9568071a1b593f6d
Lung_Opacity-1423.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1423.png,Lung_Opacity/masks/Lung_Opacity-1423.png,True,G03874,val,ddce1628cfc68a3ad504867fcb7b512f,d03d5bc34e0e4f4a
Lung_Opacity-1424.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1424.png,Lung_Opacity/masks/Lung_Opacity-1424.png,True,G03875,test,604cf59251b8e24c4d59495bd651aeaf,9179c6161acb1f96
Lung_Opacity-1425.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1425.png,Lung_Opacity/masks/Lung_Opacity-1425.png,True,G03876,test,d22ee42023d2be5a913139c0eccec2a6,9129a6041f3f3d3d
Lung_Opacity-1426.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1426.png,Lung_Opacity/masks/Lung_Opacity-1426.png,True,G03877,train,b5066949d47f8641159200af6dffb762,c23df84b4e497878
Lung_Opacity-1427.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1427.png,Lung_Opacity/masks/Lung_Opacity-1427.png,True,G03878,test,dc9a31aebbb21137694f7d647826187f,816906160f5f5f1f
Lung_Opacity-1428.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1428.png,Lung_Opacity/masks/Lung_Opacity-1428.png,True,G03879,train,d5ead87d15c40eef1c61c3eb4b74fc1d,803a5f575e5a5919
Lung_Opacity-1429.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1429.png,Lung_Opacity/masks/Lung_Opacity-1429.png,True,G03880,train,ce4e8ad08fd149b8fb7366cef1b0df76,c93997064d5d3959
Lung_Opacity-143.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-143.png,Lung_Opacity/masks/Lung_Opacity-143.png,True,G03881,train,14fccd8fe9642999580d50f105a602ff,ce054beef1b1930a
Lung_Opacity-1430.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1430.png,Lung_Opacity/masks/Lung_Opacity-1430.png,True,G03882,train,220b0d29ea48ffc6a7ce498799549d85,d93b56431e1a5939
Lung_Opacity-1431.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1431.png,Lung_Opacity/masks/Lung_Opacity-1431.png,True,G03883,train,6efafb8cd9a412cca9c2ac91022e8645,dd2952550f1f2a2d
Lung_Opacity-1432.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1432.png,Lung_Opacity/masks/Lung_Opacity-1432.png,True,G03884,train,50f96966249f16ebdb11546606210900,95698e19194f39da
Lung_Opacity-1433.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1433.png,Lung_Opacity/masks/Lung_Opacity-1433.png,True,G03885,train,08ce0336af5a0a01420defcf3213f1c9,8378145f195b5a3b
Lung_Opacity-1434.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1434.png,Lung_Opacity/masks/Lung_Opacity-1434.png,True,G03886,train,b3477ac39468ab70d605b2ec99c55fe8,8d4e0f79b0313e3a
Lung_Opacity-1435.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1435.png,Lung_Opacity/masks/Lung_Opacity-1435.png,True,G03887,train,0c7225e296ab13e1ae5040e9c16d76ea,95694ec69a5ac94e
Lung_Opacity-1436.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1436.png,Lung_Opacity/masks/Lung_Opacity-1436.png,True,G03888,val,2e151f6aa8b045ff867f1bd1671f6f68,c93a13792472e6e6
Lung_Opacity-1437.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1437.png,Lung_Opacity/masks/Lung_Opacity-1437.png,True,G03889,train,83f4d1da10d83b2521bf16cb4ae77c45,907ac74d494f4d4b
Lung_Opacity-1438.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1438.png,Lung_Opacity/masks/Lung_Opacity-1438.png,True,G03890,val,11ecd48e7300e3527dc73fd482099372,d57e861e086d4e1e
Lung_Opacity-1439.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1439.png,Lung_Opacity/masks/Lung_Opacity-1439.png,True,G03891,train,70ce3c7ee7a21e8cf45041f1743d9b09,d03b414f1e5c5b1e
Lung_Opacity-144.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-144.png,Lung_Opacity/masks/Lung_Opacity-144.png,True,G03892,val,2b834b57377fc2503815e7b48064c9aa,897f161a1a5b5b58
Lung_Opacity-1440.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1440.png,Lung_Opacity/masks/Lung_Opacity-1440.png,True,G03893,test,5cc4b1d02df7792387bf34adcbc8ef49,85231f38353c3d7a
Lung_Opacity-1441.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1441.png,Lung_Opacity/masks/Lung_Opacity-1441.png,True,G03894,test,653faaf3db6348f1fd958f74c1797bbd,817f821e19796d1d
Lung_Opacity-1442.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1442.png,Lung_Opacity/masks/Lung_Opacity-1442.png,True,G03895,train,731eedc3aca6a34e496f88c93643cde7,c075471b4d4f1f0e
Lung_Opacity-1443.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1443.png,Lung_Opacity/masks/Lung_Opacity-1443.png,True,G03896,val,94d1403f708e3e2db979c18a79a9766c,937b051d11577279
Lung_Opacity-1444.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1444.png,Lung_Opacity/masks/Lung_Opacity-1444.png,True,G03897,test,40c5ecf6c5bb72aaa3ec227f09e99d15,9817161e151c1f7f
Lung_Opacity-1445.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1445.png,Lung_Opacity/masks/Lung_Opacity-1445.png,True,G03898,train,26e53772e423f60e969f95783d966317,cd2d520f0a4f1dd9
Lung_Opacity-1446.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1446.png,Lung_Opacity/masks/Lung_Opacity-1446.png,True,G03899,train,5918857f00bbb9698922d173a1cbea33,c13b52161b173f3a
Lung_Opacity-1447.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1447.png,Lung_Opacity/masks/Lung_Opacity-1447.png,True,G03900,test,5c1d8ede812bc79a8298a9f82f48ba19,d02d4782135bcf57
Lung_Opacity-1448.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1448.png,Lung_Opacity/masks/Lung_Opacity-1448.png,True,G03901,train,f7f0555dbc116598c60ef3406acea844,d13842c71e1d6d6d
Lung_Opacity-1449.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1449.png,Lung_Opacity/masks/Lung_Opacity-1449.png,True,G03902,val,d2666d57470530b754c102106f0c8250,d42e075f4c7ce8c1
Lung_Opacity-145.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-145.png,Lung_Opacity/masks/Lung_Opacity-145.png,True,G03903,train,20d523b3aef72ae5be2263bf0c398bee,d92b460b194de93d
Lung_Opacity-1450.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1450.png,Lung_Opacity/masks/Lung_Opacity-1450.png,True,G03904,val,a387692569c62e60e677af28025fff13,d16846425b5b5e5b
Lung_Opacity-1451.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1451.png,Lung_Opacity/masks/Lung_Opacity-1451.png,True,G03905,train,6f53596d1c61edb57a1ae100bbf088ad,952c07170f4f0f5e
Lung_Opacity-1452.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1452.png,Lung_Opacity/masks/Lung_Opacity-1452.png,True,G03906,train,65b069808a49b888984d8a3b7bc5e802,ca2d0f0e4e6c6cea
Lung_Opacity-1453.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1453.png,Lung_Opacity/masks/Lung_Opacity-1453.png,True,G03907,train,8ec0a3ab8c9885dac33c933bdb9264a1,d80f430b0d2b3dbd
Lung_Opacity-1454.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1454.png,Lung_Opacity/masks/Lung_Opacity-1454.png,True,G03908,train,ad44de0ab0c46e08d10295c22423b6ee,9569069e5a39795a
Lung_Opacity-1455.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1455.png,Lung_Opacity/masks/Lung_Opacity-1455.png,True,G03909,train,40b0b27b7e52500b1e98224aa8150492,91784f1c194d3abb
Lung_Opacity-1456.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1456.png,Lung_Opacity/masks/Lung_Opacity-1456.png,True,G03910,train,b5b965a8e4dd3da4ee246436e7f619f5,95788e1f19496d6c
Lung_Opacity-1457.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1457.png,Lung_Opacity/masks/Lung_Opacity-1457.png,True,G03911,train,ff8bd30bb44d81bb33b5b66b8385b357,891a165a587b7f61
Lung_Opacity-1458.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1458.png,Lung_Opacity/masks/Lung_Opacity-1458.png,True,G03912,train,e7491903f6afca7e1ae6280c9a64e650,c87fd242d65b5642
Lung_Opacity-1459.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1459.png,Lung_Opacity/masks/Lung_Opacity-1459.png,True,G03913,train,d9ff652823162fbba0e840e8e0fecb98,cd6f961a1e425b58
Lung_Opacity-146.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-146.png,Lung_Opacity/masks/Lung_Opacity-146.png,True,G03914,train,d41480e0047d7ccc53a2a9d222574c78,d42e455a28cd1cfd
Lung_Opacity-1460.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1460.png,Lung_Opacity/masks/Lung_Opacity-1460.png,True,G03915,train,00b30166aa75b0e163bf85be28fe74ff,d43bc71f0e5b190a
Lung_Opacity-1461.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1461.png,Lung_Opacity/masks/Lung_Opacity-1461.png,True,G03916,test,bbbf80c087e845bbcf4e5108640c2d8f,c12d060f4b3d5a7d
Lung_Opacity-1462.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1462.png,Lung_Opacity/masks/Lung_Opacity-1462.png,True,G03917,test,13b1f2edc05796b02893c31baf432c1e,c52d1e425b7fe828
Lung_Opacity-1463.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1463.png,Lung_Opacity/masks/Lung_Opacity-1463.png,True,G03918,train,3793201d75ee47bd1a04bb1efcc4a62e,897b86161ecdb8d8
Lung_Opacity-1464.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1464.png,Lung_Opacity/masks/Lung_Opacity-1464.png,True,G03919,train,1af6e482c69c703d8a7938704a8974dd,dd2a520d8e683db6
Lung_Opacity-1465.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1465.png,Lung_Opacity/masks/Lung_Opacity-1465.png,True,G03920,train,22a817a67bd80482bc858f2bbf5b8ce9,942d571a1b5b1979
Lung_Opacity-1466.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1466.png,Lung_Opacity/masks/Lung_Opacity-1466.png,True,G03921,train,5bb46a2ecda62e7ae357531a6db34b71,d01a0b1fbe96d2e4
Lung_Opacity-1467.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1467.png,Lung_Opacity/masks/Lung_Opacity-1467.png,True,G03922,train,bb0bb2829cb1b94ab5e8a419a97a8e08,d13b521a1b0b3fb8
Lung_Opacity-1468.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1468.png,Lung_Opacity/masks/Lung_Opacity-1468.png,True,G03923,train,fa187362a5fb159ff61571b16b39ed94,91685f1a5979e3a4
Lung_Opacity-1469.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1469.png,Lung_Opacity/masks/Lung_Opacity-1469.png,True,G03924,train,3c7b438627b333c61c8856624e9bebb7,dd2e421b38e4edb0
Lung_Opacity-147.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-147.png,Lung_Opacity/masks/Lung_Opacity-147.png,True,G03925,train,7713931111841a00b530e61511a0d8e5,817f061a1d5e1b57
Lung_Opacity-1470.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1470.png,Lung_Opacity/masks/Lung_Opacity-1470.png,True,G03926,train,a037f72f8274070942d5fa2b29cd6001,d42fc30d4c1e6e4e
Lung_Opacity-1471.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1471.png,Lung_Opacity/masks/Lung_Opacity-1471.png,True,G03927,train,37ad00aa734ff15d2b2cf10de1574afd,d42ec15d0e2f6c6a
Lung_Opacity-1472.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1472.png,Lung_Opacity/masks/Lung_Opacity-1472.png,True,G03928,train,4c874f027710e34c37221e0ce36a06bd,c03b520a3e2b3f3b
Lung_Opacity-1473.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1473.png,Lung_Opacity/masks/Lung_Opacity-1473.png,True,G03929,train,243c41ae86c19a5695f8678cd341149c,d42c43cb0f0d4f4f
Lung_Opacity-1474.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1474.png,Lung_Opacity/masks/Lung_Opacity-1474.png,True,G03930,val,a07bc1059c276ce98e30669306feac8e,d47ec3d39b41d01c
Lung_Opacity-1475.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1475.png,Lung_Opacity/masks/Lung_Opacity-1475.png,True,G03931,val,7d532bf9a4a397a619c035610c94f126,c13bd64f58197949
Lung_Opacity-1476.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1476.png,Lung_Opacity/masks/Lung_Opacity-1476.png,True,G03932,val,5cf7200a455e1bc0a408fbdec6d880bb,997b8a1f0a4f384e
Lung_Opacity-1477.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1477.png,Lung_Opacity/masks/Lung_Opacity-1477.png,True,G03933,train,dbdf2e55c06085794cdc5acf6e33d369,913bc7969eca98a8
Lung_Opacity-1478.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1478.png,Lung_Opacity/masks/Lung_Opacity-1478.png,True,G03934,test,2bd1f171894224880a8290f684c90e20,dc2f410fce0a0f4f
Lung_Opacity-1479.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1479.png,Lung_Opacity/masks/Lung_Opacity-1479.png,True,G03935,test,62382088f51fb8bde775820765a731cf,c8355d1a4b267d3c
Lung_Opacity-148.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-148.png,Lung_Opacity/masks/Lung_Opacity-148.png,True,G03936,train,6b8e234b0b478d2acd887289b8d36c1a,817c96121a1f793f
Lung_Opacity-1480.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1480.png,Lung_Opacity/masks/Lung_Opacity-1480.png,True,G03937,test,39e710d0a659a2bb580e6be33e16e0ed,817d141a195b5b5f
Lung_Opacity-1481.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1481.png,Lung_Opacity/masks/Lung_Opacity-1481.png,True,G03938,train,933c6abc6743897c89b0d977deaa2a05,95684e561b4d4f39
Lung_Opacity-1482.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1482.png,Lung_Opacity/masks/Lung_Opacity-1482.png,True,G03939,train,0b041b7810c85b58df68ffe9bc7c73b3,997886160e1f1f1f
Lung_Opacity-1483.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1483.png,Lung_Opacity/masks/Lung_Opacity-1483.png,True,G03940,train,adee6f53dc51bf5e4741dd3a20615c66,807f84564a5d5f4e
Lung_Opacity-1484.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1484.png,Lung_Opacity/masks/Lung_Opacity-1484.png,True,G03941,train,d8b4b31e4cc4f980accbea46217a89a5,95000f571e4fae7c
Lung_Opacity-1485.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1485.png,Lung_Opacity/masks/Lung_Opacity-1485.png,True,G03942,train,39fca408ce6503eee7ae6f2556e86a39,9168065f5a5d5c7a
Lung_Opacity-1486.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1486.png,Lung_Opacity/masks/Lung_Opacity-1486.png,True,G03943,train,3075a5feff5189282c28a88a64953ae0,cd7c92591943397c
Lung_Opacity-1487.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1487.png,Lung_Opacity/masks/Lung_Opacity-1487.png,True,G03944,test,c166fb312668a518cedc1c097c63ce8a,d03a431b38783dd7
Lung_Opacity-1488.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1488.png,Lung_Opacity/masks/Lung_Opacity-1488.png,True,G03945,train,983b8f3bb3a762302b235d336250db96,807d060b5b4d5f5d
Lung_Opacity-1489.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1489.png,Lung_Opacity/masks/Lung_Opacity-1489.png,True,G03946,train,6a53fec103a52478985931b04031a6e6,9129ce5c3a5d3959
Lung_Opacity-149.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-149.png,Lung_Opacity/masks/Lung_Opacity-149.png,True,G03947,train,571b4d94a27980e5211bc32679e73961,98992fd22327d678
Lung_Opacity-1490.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1490.png,Lung_Opacity/masks/Lung_Opacity-1490.png,True,G03948,train,9cbc62762c2ace7d3ef6639ce55c92bf,dd0e420d0c4aefed
Lung_Opacity-1491.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1491.png,Lung_Opacity/masks/Lung_Opacity-1491.png,True,G03949,train,cd60e43263019550c79f7b0d43cd971a,d02d4b3b70b3b30b
Lung_Opacity-1492.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1492.png,Lung_Opacity/masks/Lung_Opacity-1492.png,True,G03950,val,c7d9c372d64c91d041cfe6afcb55260d,9b44161a1d5a5f5b
Lung_Opacity-1493.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1493.png,Lung_Opacity/masks/Lung_Opacity-1493.png,True,G03951,train,2345f84566142081e253a9b7a112afc1,d96c861c59d668e9
Lung_Opacity-1494.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1494.png,Lung_Opacity/masks/Lung_Opacity-1494.png,True,G03952,test,071c0c1f8d43333b56d635cd1b488303,dc3dd34b0f2c0e0d
Lung_Opacity-1495.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1495.png,Lung_Opacity/masks/Lung_Opacity-1495.png,True,G03953,train,f76a2fdb766ec70ee1acf0ce891fd39a,c52a934d650e4f4f
Lung_Opacity-1496.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1496.png,Lung_Opacity/masks/Lung_Opacity-1496.png,True,G03954,val,c63fb169c8c25f55c177aae03c7c1aca,883dc71a1f4e1a6d
Lung_Opacity-1497.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1497.png,Lung_Opacity/masks/Lung_Opacity-1497.png,True,G03955,val,8ad88d10081d2cd4cc654fbbe9d7d851,d02f46505b1a5f4f
Lung_Opacity-1498.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1498.png,Lung_Opacity/masks/Lung_Opacity-1498.png,True,G03956,test,25acd2f018330f72f0e7ef84f18b8c74,c93d861c5179bce1
Lung_Opacity-1499.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1499.png,Lung_Opacity/masks/Lung_Opacity-1499.png,True,G03957,train,17576637b96d68a065ac2b1e172b1c0c,d929961e1d595959
Lung_Opacity-15.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-15.png,Lung_Opacity/masks/Lung_Opacity-15.png,True,G03958,train,b81f3b5ebba70e901514e41b95c4769d,d07eaf1f1c4d0d0c
Lung_Opacity-150.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-150.png,Lung_Opacity/masks/Lung_Opacity-150.png,True,G03959,train,b728c2eae2077e45ac3ef99bbd95f73e,954a5ef121b53c3c
Lung_Opacity-1500.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1500.png,Lung_Opacity/masks/Lung_Opacity-1500.png,True,G03960,train,7a61953180cea5aae01ce9aa8cd2e7f0,8b68065a5a5f595d
Lung_Opacity-1501.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1501.png,Lung_Opacity/masks/Lung_Opacity-1501.png,True,G03961,train,b0da7c0538f87932c08c951fa304ed6c,c03d53564a4b6e2f
Lung_Opacity-1502.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1502.png,Lung_Opacity/masks/Lung_Opacity-1502.png,True,G03962,train,4369426251c360c7a75e0afc52a3f5b7,997847071717471f
Lung_Opacity-1503.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1503.png,Lung_Opacity/masks/Lung_Opacity-1503.png,True,G03963,val,fa54aa450e3cf1ac400202080e82851e,9538071f191f1f5c
Lung_Opacity-1504.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1504.png,Lung_Opacity/masks/Lung_Opacity-1504.png,True,G03964,val,f67db47668199e6f5b9f297974921140,d07f030b0d1e2d7e
Lung_Opacity-1505.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1505.png,Lung_Opacity/masks/Lung_Opacity-1505.png,True,G03965,train,99a9b0519a8e68edae8a38682749c0e6,c0295a96c75e1e5b
Lung_Opacity-1506.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1506.png,Lung_Opacity/masks/Lung_Opacity-1506.png,True,G03966,train,d3a14ca8dc790b0db45a5e0377da5b2d,c42f47472a6961fc
Lung_Opacity-1507.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1507.png,Lung_Opacity/masks/Lung_Opacity-1507.png,True,G03967,train,5c330367bdac3e5eb45e545277f7d109,d23d421b1e1e1f1d
Lung_Opacity-1508.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1508.png,Lung_Opacity/masks/Lung_Opacity-1508.png,True,G03968,test,4c55fc470390a630d305a22d109c21e1,813b1d4e1f58785b
Lung_Opacity-1509.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1509.png,Lung_Opacity/masks/Lung_Opacity-1509.png,True,G03969,val,6b5a7011cd88513c27c21dd59bff45e2,c83917de1c985b9c
Lung_Opacity-151.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-151.png,Lung_Opacity/masks/Lung_Opacity-151.png,True,G03970,train,edb0bd282c6b1ab436368c56dc8575c0,c42f834b4c7c6d6c
Lung_Opacity-1510.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1510.png,Lung_Opacity/masks/Lung_Opacity-1510.png,True,G03971,test,b8ba2da3c921955c3ca485351feb514a,894f061b0b4f4f4f
Lung_Opacity-1511.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1511.png,Lung_Opacity/masks/Lung_Opacity-1511.png,True,G03972,test,56ca5183a9eb6abb879dba8c56e8bb84,cd7e87164b461a4b
Lung_Opacity-1512.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1512.png,Lung_Opacity/masks/Lung_Opacity-1512.png,True,G03973,test,3f48c11423aa95b35489f9a5090187c8,c57ee2c1d1c9a985
Lung_Opacity-1513.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1513.png,Lung_Opacity/masks/Lung_Opacity-1513.png,True,G03974,train,66230a690ce87fbfc095c8d0421937df,97600c1f5f58195f
Lung_Opacity-1514.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1514.png,Lung_Opacity/masks/Lung_Opacity-1514.png,True,G03975,test,ddf28b0efb6fbd298bfd3bb3e754b2eb,991906d61a594d7f
Lung_Opacity-1515.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1515.png,Lung_Opacity/masks/Lung_Opacity-1515.png,True,G03976,train,e1dc5da4809b5469097e20f57fb53ed0,c03e2f0d091f9b9b
Lung_Opacity-1516.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1516.png,Lung_Opacity/masks/Lung_Opacity-1516.png,True,G03977,train,bd71a847f232ec1f7a3cef84367cb90e,cc3c13431e6d7c69
Lung_Opacity-1517.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1517.png,Lung_Opacity/masks/Lung_Opacity-1517.png,True,G03978,train,5ae5a6779c49f51ff9ef11712c21167a,912bd70831595f37
Lung_Opacity-1518.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1518.png,Lung_Opacity/masks/Lung_Opacity-1518.png,True,G03979,train,2a2b5e0e4eeb4e32ba30e2b9ac38c0c4,dc6c07590e6c2e7a
Lung_Opacity-1519.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1519.png,Lung_Opacity/masks/Lung_Opacity-1519.png,True,G03980,train,27723eb552d0343ecdd6c21627d0f03c,817a061e79686fe9
Lung_Opacity-152.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-152.png,Lung_Opacity/masks/Lung_Opacity-152.png,True,G03981,train,9e6d927ae8bf85229ca1e0a53a6e3b67,d42d52535f4d5c4c
Lung_Opacity-1520.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1520.png,Lung_Opacity/masks/Lung_Opacity-1520.png,True,G03982,train,3789df340a714797ce3b5cc01b96030c,893b969a190d1d3f
Lung_Opacity-1521.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1521.png,Lung_Opacity/masks/Lung_Opacity-1521.png,True,G03983,train,374dcd12a55eb966d57c3a2159370971,de2d590e1f2c9391
Lung_Opacity-1522.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1522.png,Lung_Opacity/masks/Lung_Opacity-1522.png,True,G03984,train,ad18967d2702fd7a711a69fe918b58cc,8168c4545a5f5f5b
Lung_Opacity-1523.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1523.png,Lung_Opacity/masks/Lung_Opacity-1523.png,True,G03985,train,239aa5e8b9ecc7dc37274b2861462fe7,807dd7d281c25773
Lung_Opacity-1524.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1524.png,Lung_Opacity/masks/Lung_Opacity-1524.png,True,G03986,train,9cdafe024507502c8ed9f4a0cbe175a6,956e86861b0f1f1e
Lung_Opacity-1525.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1525.png,Lung_Opacity/masks/Lung_Opacity-1525.png,True,G03987,test,2405ad9ee78f8dbb5e9c7a08d45f3590,d02dd5c34b4f2d2c
Lung_Opacity-1526.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1526.png,Lung_Opacity/masks/Lung_Opacity-1526.png,True,G03988,train,7effbb7ef4e2c7ad85280eec500f752b,91784f471c6de13a
Lung_Opacity-1527.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1527.png,Lung_Opacity/masks/Lung_Opacity-1527.png,True,G03989,train,6be72154547baa2b4440d1f4ba55676b,cc6cd2575f51485c
Lung_Opacity-1528.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1528.png,Lung_Opacity/masks/Lung_Opacity-1528.png,True,G03990,train,a9a86279f2fd5752d6064ced368126a2,9c2cc21b8f571b1d
Lung_Opacity-1529.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1529.png,Lung_Opacity/masks/Lung_Opacity-1529.png,True,G03991,train,66c3090f4751c29cd8f1280f8278d05d,f95947471d59183c
Lung_Opacity-153.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-153.png,Lung_Opacity/masks/Lung_Opacity-153.png,True,G03992,train,b6a03ef9bb4b33bab5b32056875d6a04,c8644f4d4e2c6e6d
Lung_Opacity-1530.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1530.png,Lung_Opacity/masks/Lung_Opacity-1530.png,True,G03993,train,d4b3a38fbec011111bb8558a666318d7,dd3c82539a13db43
Lung_Opacity-1531.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1531.png,Lung_Opacity/masks/Lung_Opacity-1531.png,True,G03994,train,1ee81880bf516c162a9e9a099fb7385c,d42f470d0a2e2e7e
Lung_Opacity-1532.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1532.png,Lung_Opacity/masks/Lung_Opacity-1532.png,True,G03995,val,4dfd4057665b44c25c69fef55fc9d0bc,c87b849e1c5d115f
Lung_Opacity-1533.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1533.png,Lung_Opacity/masks/Lung_Opacity-1533.png,True,G03996,val,023c145bd0f52536b178dba311a84dbc,cd2e02df0c6cee4a
Lung_Opacity-1534.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1534.png,Lung_Opacity/masks/Lung_Opacity-1534.png,True,G03997,val,1774f992adf05ad27e0ba91c0b9e70f2,ca3745c8492e6e6e
Lung_Opacity-1535.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1535.png,Lung_Opacity/masks/Lung_Opacity-1535.png,True,G03998,test,7312f28e8c4e79aeee1339083f1a6ee4,d42f434b4f4e0f0e
Lung_Opacity-1536.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1536.png,Lung_Opacity/masks/Lung_Opacity-1536.png,True,G03999,train,5cd3f9bb0555f65e434115bf93e2ed3a,95780f1e594d48eb
Lung_Opacity-1537.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1537.png,Lung_Opacity/masks/Lung_Opacity-1537.png,True,G04000,train,147c0e7de82ac06d32f25de05458487a,95e886171f1d1d4d
Lung_Opacity-1538.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1538.png,Lung_Opacity/masks/Lung_Opacity-1538.png,True,G04001,train,a699cd01066b4bcdcc477927bd869b43,8179869e5b8a5b1b
Lung_Opacity-1539.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1539.png,Lung_Opacity/masks/Lung_Opacity-1539.png,True,G04002,train,7ce5e1172d61f02fd48d91fc7c675b4f,c43f820b4c666ee7
Lung_Opacity-154.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-154.png,Lung_Opacity/masks/Lung_Opacity-154.png,True,G04003,train,76ffb00dbee721a98d751cc639afadf6,856e82530f7c5f68
Lung_Opacity-1540.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1540.png,Lung_Opacity/masks/Lung_Opacity-1540.png,True,G04004,train,505c039c7eb17cfce93aaca8e66c64a0,817b061a195f1f57
Lung_Opacity-1541.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1541.png,Lung_Opacity/masks/Lung_Opacity-1541.png,True,G04005,train,6c9639864b3d04cf541b433a6bdef82c,c03d5c090f0f6d6f
Lung_Opacity-1542.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1542.png,Lung_Opacity/masks/Lung_Opacity-1542.png,True,G04006,train,0b11ef8df050d21b6b3cd58f7150dbe3,856c07471f5d4b4d
Lung_Opacity-1543.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1543.png,Lung_Opacity/masks/Lung_Opacity-1543.png,True,G04007,train,b4132468de9d5692e3b939cac37f1662,9578475a186d2e3e
Lung_Opacity-1544.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1544.png,Lung_Opacity/masks/Lung_Opacity-1544.png,True,G04008,train,66ce87b2043b3821a242c63fb59feed5,98fd851c0b1d0d7d
Lung_Opacity-1545.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1545.png,Lung_Opacity/masks/Lung_Opacity-1545.png,True,G04009,train,43d433fca1c1838ae684f2d87be57b34,d07dc21b0a6f3b58
Lung_Opacity-1546.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1546.png,Lung_Opacity/masks/Lung_Opacity-1546.png,True,G04010,train,17d18ee2799d01c546cc2687a08d61bd,9c2a431f471e465f
Lung_Opacity-1547.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1547.png,Lung_Opacity/masks/Lung_Opacity-1547.png,True,G04011,train,b0c2c816557a0023400b2b04c06960e7,cd2cd2524f491e1f
Lung_Opacity-1548.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1548.png,Lung_Opacity/masks/Lung_Opacity-1548.png,True,G04012,train,4af134e610faca7618f712fcc384a2d5,d92b121979383d3d
Lung_Opacity-1549.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1549.png,Lung_Opacity/masks/Lung_Opacity-1549.png,True,G04013,val,eedc9fdee83320845931c53c0aa98f1d,d02d5f574a4a6a6a
Lung_Opacity-155.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-155.png,Lung_Opacity/masks/Lung_Opacity-155.png,True,G04014,val,6322310d6740c3ed8a3839a216433fe7,dc0c131f1d1a1f5b
Lung_Opacity-1550.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1550.png,Lung_Opacity/masks/Lung_Opacity-1550.png,True,G04015,test,107a878cd86d92e6754c6f549ae6f173,9368c016fa5df859
Lung_Opacity-1551.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1551.png,Lung_Opacity/masks/Lung_Opacity-1551.png,True,G04016,test,bc67d66e83c26f9f8af3216688ff5cf1,c939575a174e5949
Lung_Opacity-1552.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1552.png,Lung_Opacity/masks/Lung_Opacity-1552.png,True,G04017,test,41db12627676acdd911b94f3cca7ecc5,803f560a4f0e5f2f
Lung_Opacity-1553.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1553.png,Lung_Opacity/masks/Lung_Opacity-1553.png,True,G04018,val,368e948caa5ebf721c17d86dc5e92052,c53c12591d5a697d
Lung_Opacity-1554.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1554.png,Lung_Opacity/masks/Lung_Opacity-1554.png,True,G04019,train,1c548330681a7ad799aea8050e3cba58,dcbdc24f424f6c42
Lung_Opacity-1555.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1555.png,Lung_Opacity/masks/Lung_Opacity-1555.png,True,G04020,train,9ffa622c72a79342e70647e3f54a5c4d,93385e585a4b89cf
Lung_Opacity-1556.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1556.png,Lung_Opacity/masks/Lung_Opacity-1556.png,True,G04021,train,76762b2105a0babdf88c0626c17b92cd,d12a1f1e5a597943
Lung_Opacity-1557.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1557.png,Lung_Opacity/masks/Lung_Opacity-1557.png,True,G04022,train,609f653609d9deebd5941f8bc046d471,dc3b43272c3475d1
Lung_Opacity-1558.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1558.png,Lung_Opacity/masks/Lung_Opacity-1558.png,True,G04023,train,ef347a645951912f3baeb783ca720a2f,ec7cd34b454b184e
Lung_Opacity-1559.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1559.png,Lung_Opacity/masks/Lung_Opacity-1559.png,True,G04024,train,cd9ccaf27165813da4652e5ba3ee78bb,d56e831d0c6c2e4f
Lung_Opacity-156.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-156.png,Lung_Opacity/masks/Lung_Opacity-156.png,True,G04025,test,43ea040ccfa34cced4050ce502dfc61d,cc3d4b0b2e3233b3
Lung_Opacity-1560.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1560.png,Lung_Opacity/masks/Lung_Opacity-1560.png,True,G04026,train,b3d8bf28a3c31645c75abacc8c18c26c,c13c6b3e36a392d1
Lung_Opacity-1561.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1561.png,Lung_Opacity/masks/Lung_Opacity-1561.png,True,G04027,train,11d1ed413c9353b4ca4b1ccfb0efb147,897f861e190d436f
Lung_Opacity-1562.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1562.png,Lung_Opacity/masks/Lung_Opacity-1562.png,True,G04028,train,04637c420b449101935e0b2534d127d9,d4b5434a9b424f1f
Lung_Opacity-1563.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1563.png,Lung_Opacity/masks/Lung_Opacity-1563.png,True,G04029,train,ab1c2ada2fa4715be38019d26413c589,cc4f030c2c7ef3b8
Lung_Opacity-1564.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1564.png,Lung_Opacity/masks/Lung_Opacity-1564.png,True,G04030,train,86acaa4c8aadfaf6af67ec73497399ad,d918161e585a7ee3
Lung_Opacity-1565.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1565.png,Lung_Opacity/masks/Lung_Opacity-1565.png,True,G04031,train,9e305d2970ba3aabb778f3793104f1d3,813b56696f2e3c2c
Lung_Opacity-1566.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1566.png,Lung_Opacity/masks/Lung_Opacity-1566.png,True,G04032,train,3a7363ca424005f4283e3f52c2d57377,c12bf506de1a5f18
Lung_Opacity-1567.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1567.png,Lung_Opacity/masks/Lung_Opacity-1567.png,True,G04033,train,717f526210ffd7f56633e331e4c32a1a,dc2ec71f4d0c070f
Lung_Opacity-1568.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1568.png,Lung_Opacity/masks/Lung_Opacity-1568.png,True,G04034,val,64486064ff95f92548d8bf90cb284b31,cc3d03431f4e4d3d
Lung_Opacity-1569.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1569.png,Lung_Opacity/masks/Lung_Opacity-1569.png,True,G04035,val,955c894129168c791b28d49cb8ea98da,c92b061a194f5b6f
Lung_Opacity-157.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-157.png,Lung_Opacity/masks/Lung_Opacity-157.png,True,G04036,val,efa232fc99964ad0d21030ec0fca0e36,c93817525b4f4f4c
Lung_Opacity-1570.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1570.png,Lung_Opacity/masks/Lung_Opacity-1570.png,True,G04037,test,bdc4db61194c9b7704ce4b09e073b4f5,d90a5e085d2de5a7
Lung_Opacity-1571.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1571.png,Lung_Opacity/masks/Lung_Opacity-1571.png,True,G04038,train,d33aa77ada4ac55f025ad596bd19bca4,8362153f1d5c783b
Lung_Opacity-1572.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1572.png,Lung_Opacity/masks/Lung_Opacity-1572.png,True,G04039,test,ce716406bfcc5f95ad73252c428451c3,917b86969a4f391c
Lung_Opacity-1573.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1573.png,Lung_Opacity/masks/Lung_Opacity-1573.png,True,G04040,train,928abf4e7227e79fb14bcd5781f4781b,d929c71e5859193b
Lung_Opacity-1574.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1574.png,Lung_Opacity/masks/Lung_Opacity-1574.png,True,G04041,train,30e0f4241ff5146edcbbcfe4c272087d,d429c3575d6a583a
Lung_Opacity-1575.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1575.png,Lung_Opacity/masks/Lung_Opacity-1575.png,True,G04042,train,fd19f92a6f8225319d188d32cdb9d9d5,81e96a947ec2d87a
Lung_Opacity-1576.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1576.png,Lung_Opacity/masks/Lung_Opacity-1576.png,True,G04043,train,ea01f949c73e257eb39b846a92d9c430,d42b531e082d3dbd
Lung_Opacity-1577.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1577.png,Lung_Opacity/masks/Lung_Opacity-1577.png,True,G04044,train,956faed27369aba035f9c0ecbce720cb,917a125b3b3b31a3
Lung_Opacity-1578.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1578.png,Lung_Opacity/masks/Lung_Opacity-1578.png,True,G04045,train,49d42ed71e125e6cd970c49cf63eed2a,d93c461b1d4d38f1
Lung_Opacity-1579.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1579.png,Lung_Opacity/masks/Lung_Opacity-1579.png,True,G04046,train,51765a3dbb54fba6eeb033f2191aae5b,91780e1b596979e5
Lung_Opacity-158.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-158.png,Lung_Opacity/masks/Lung_Opacity-158.png,True,G04047,train,d6a8f31bdbc7b81703fd5da51a850aff,816f841e9e1e1e5b
Lung_Opacity-1580.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1580.png,Lung_Opacity/masks/Lung_Opacity-1580.png,True,G04048,train,e0e589613998809f847a65bd075807eb,dd2e42830b4dac3f
Lung_Opacity-1581.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1581.png,Lung_Opacity/masks/Lung_Opacity-1581.png,True,G04049,test,19000f7c516440b0ddcc803ebb33960d,807f87161a1f5c5e
Lung_Opacity-1582.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1582.png,Lung_Opacity/masks/Lung_Opacity-1582.png,True,G04050,test,dd217910a5aab8236eec8c9a26075b15,946c074b5f4d2c7c
Lung_Opacity-1583.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1583.png,Lung_Opacity/masks/Lung_Opacity-1583.png,True,G04051,train,7911db312109bf9ce41b62d79d5a342c,c43d5b4b0c2c3cbe
Lung_Opacity-1584.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1584.png,Lung_Opacity/masks/Lung_Opacity-1584.png,True,G04052,train,ec3a4a0be99201fa09cd8656473cab87,947f471a0f481b6b
Lung_Opacity-1585.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1585.png,Lung_Opacity/masks/Lung_Opacity-1585.png,True,G04053,train,7abb50ce3db52f63a81fa6ecf5f5c61f,995f061a595d7c68
Lung_Opacity-1586.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1586.png,Lung_Opacity/masks/Lung_Opacity-1586.png,True,G04054,train,728c137f4e43fc6902df355c1249ec83,d52ec34b4e0c4f4d
Lung_Opacity-1587.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1587.png,Lung_Opacity/masks/Lung_Opacity-1587.png,True,G04055,val,0e6cac39cd045ea791b32f80fd12d889,993b46c6561ecd49
Lung_Opacity-1588.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1588.png,Lung_Opacity/masks/Lung_Opacity-1588.png,True,G04056,val,71afac55c883621ae309094cf48d0808,816d86161f1e5e5d
Lung_Opacity-1589.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1589.png,Lung_Opacity/masks/Lung_Opacity-1589.png,True,G04057,val,50418c4813ed3f940e8328d3eb452c66,85681f869a189ddf
Lung_Opacity-159.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-159.png,Lung_Opacity/masks/Lung_Opacity-159.png,True,G04058,test,98d33e839ff070ce0e4311771e023d64,d52f42d35b4d4c4c
Lung_Opacity-1590.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1590.png,Lung_Opacity/masks/Lung_Opacity-1590.png,True,G04059,train,8fb3c30fcf4675f4fc617ed3f5896cee,c03dc7171d0f1a5e
Lung_Opacity-1591.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1591.png,Lung_Opacity/masks/Lung_Opacity-1591.png,True,G04060,train,3ba4c5d7e6bed42c2c6afc5f92994bcc,d42d434f0c6e6e6a
Lung_Opacity-1592.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1592.png,Lung_Opacity/masks/Lung_Opacity-1592.png,True,G04061,train,8baa4e58698fad58ba60bc83d1b9e5e0,d94f865a496be5a0
Lung_Opacity-1593.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1593.png,Lung_Opacity/masks/Lung_Opacity-1593.png,True,G04062,train,180656fe4a86355bcbe1c628e680fe15,d46e4f1d0a2e2e35
Lung_Opacity-1594.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1594.png,Lung_Opacity/masks/Lung_Opacity-1594.png,True,G04063,train,55f9a76cda43c8e22446b994fb8e618b,cd6d56c3530b4f0a
Lung_Opacity-1595.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1595.png,Lung_Opacity/masks/Lung_Opacity-1595.png,True,G04064,train,9d6c7165efd81a602c9312609fa1f917,817b941a391deab9
Lung_Opacity-1596.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1596.png,Lung_Opacity/masks/Lung_Opacity-1596.png,True,G04065,train,e206eb8639aa58f24e1240040d190186,857d0f4b484d4f4d
Lung_Opacity-1597.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1597.png,Lung_Opacity/masks/Lung_Opacity-1597.png,True,G04066,train,0816d3291cab6986bd5276ec6267ce64,897e8f0a5d6c4c55
Lung_Opacity-1598.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1598.png,Lung_Opacity/masks/Lung_Opacity-1598.png,True,G04067,train,0283fd963f7cdfe1a4f426a3b1b10541,c83d531c2e794e1d
Lung_Opacity-1599.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1599.png,Lung_Opacity/masks/Lung_Opacity-1599.png,True,G04068,train,9a288f339d4618d7fd8a35a236892eb8,dc2d530b0d382e3f
Lung_Opacity-16.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-16.png,Lung_Opacity/masks/Lung_Opacity-16.png,True,G04069,train,1ef87004473a9482fa6deaa03288ce4b,d43d430d18adfcf0
Lung_Opacity-160.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-160.png,Lung_Opacity/masks/Lung_Opacity-160.png,True,G04070,train,75f31331ecdc62f16cc0e18058aeb39b,c9691597165b0e5b
Lung_Opacity-1600.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1600.png,Lung_Opacity/masks/Lung_Opacity-1600.png,True,G04071,val,97c793daa220b62c47037a6d321039c2,883d86c61f1e1e5b
Lung_Opacity-1601.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1601.png,Lung_Opacity/masks/Lung_Opacity-1601.png,True,G04072,val,bd488b07f1414836273d4ef4f0d0a0fb,d52a57570f0c2d0f
Lung_Opacity-1602.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1602.png,Lung_Opacity/masks/Lung_Opacity-1602.png,True,G04073,train,372a6cc225ed5d60c936bf4ea8c6b048,d03b121b3bbbb780
Lung_Opacity-1603.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1603.png,Lung_Opacity/masks/Lung_Opacity-1603.png,True,G04074,test,b19743fe2c6ff2b3f21d974246029c3f,d929461f3a39392d
Lung_Opacity-1604.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1604.png,Lung_Opacity/masks/Lung_Opacity-1604.png,True,G04075,train,bd433d06132fc0cbbd24e6f8a317f44d,c96f161a195f181f
Lung_Opacity-1605.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1605.png,Lung_Opacity/masks/Lung_Opacity-1605.png,True,G04076,test,53d4ab78b82463a28215407dae70596b,95784fc21e5be948
Lung_Opacity-1606.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1606.png,Lung_Opacity/masks/Lung_Opacity-1606.png,True,G04077,val,ffca5478ce4c1667b788c8bb7925a4fe,85681c1f5f595a5a
Lung_Opacity-1607.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1607.png,Lung_Opacity/masks/Lung_Opacity-1607.png,True,G04078,test,94d61c61bceea6cd17ef3fe69424cc9d,817b96161c5e5b59
Lung_Opacity-1608.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1608.png,Lung_Opacity/masks/Lung_Opacity-1608.png,True,G04079,train,e0385ecfc9ea169290f94edcb4e18fab,981945c7961e5b79
Lung_Opacity-1609.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1609.png,Lung_Opacity/masks/Lung_Opacity-1609.png,True,G04080,train,3ab41d29b24baf33850722fcd9a31643,c43dd7434b0b2f0d
Lung_Opacity-161.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-161.png,Lung_Opacity/masks/Lung_Opacity-161.png,True,G04081,train,37e1282cc299d655e56547ef908d72b4,cc2df3d3c283c345
Lung_Opacity-1610.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1610.png,Lung_Opacity/masks/Lung_Opacity-1610.png,True,G04082,train,e4a0bc36049dfc1b3bd8f20c75bafd7b,dd2a071f0d1d1e4e
Lung_Opacity-1611.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1611.png,Lung_Opacity/masks/Lung_Opacity-1611.png,True,G04083,train,ae55bffe4aadaae0050b96d0bcd95eab,8d4ad34d2c0eadb5
Lung_Opacity-1612.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1612.png,Lung_Opacity/masks/Lung_Opacity-1612.png,True,G04084,train,18a88d8dcd3c942d91b5c418132c2615,d42f455f0e0e2c6e
Lung_Opacity-1613.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1613.png,Lung_Opacity/masks/Lung_Opacity-1613.png,True,G04085,train,91407f414ae20b081c1b426bc4713816,c43d12c3e34b4f4d
Lung_Opacity-1614.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1614.png,Lung_Opacity/masks/Lung_Opacity-1614.png,True,G04086,train,e493be50fe967beb65268b365bead350,ff3be7121a414319
Lung_Opacity-1615.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1615.png,Lung_Opacity/masks/Lung_Opacity-1615.png,True,G04087,train,26d473c96b59945376c3d89595ce152c,c52e43430f4c6f4f
Lung_Opacity-1616.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1616.png,Lung_Opacity/masks/Lung_Opacity-1616.png,True,G04088,val,1ccfbb8e5a549f935a32a224617f7cb1,996fc61e0b4c1e1d
Lung_Opacity-1617.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1617.png,Lung_Opacity/masks/Lung_Opacity-1617.png,True,G04089,train,ccd5a4903c04fc962286547398a08a90,c53f16131a6c1b4f
Lung_Opacity-1618.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1618.png,Lung_Opacity/masks/Lung_Opacity-1618.png,True,G04090,train,3db8d77e7842d840cdda621bbe6f51e0,d43d53434f4b4b1a
Lung_Opacity-1619.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1619.png,Lung_Opacity/masks/Lung_Opacity-1619.png,True,G04091,train,b8e83ca37727c7a08b9807c7a51f2dd9,d42e434f0e4d2d2f
Lung_Opacity-162.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-162.png,Lung_Opacity/masks/Lung_Opacity-162.png,True,G04092,train,243d2e831aba176cb008392d93fcb48b,cc3c430b986d3cfc
Lung_Opacity-1620.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1620.png,Lung_Opacity/masks/Lung_Opacity-1620.png,True,G04093,train,320480941793c430cfa61d3c3d523ba2,8c3ed30f0e1e0f2d
Lung_Opacity-1621.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1621.png,Lung_Opacity/masks/Lung_Opacity-1621.png,True,G04094,val,092a45336bddc3ddccde96cbc63d1dad,dd28c3174d4d1d4d
Lung_Opacity-1622.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1622.png,Lung_Opacity/masks/Lung_Opacity-1622.png,True,G04095,val,5bbff8c37a7d333dde28b6b6ddd95b85,8578a6d35b87161b
Lung_Opacity-1623.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1623.png,Lung_Opacity/masks/Lung_Opacity-1623.png,True,G04096,test,ea5549bf4fc34bdd0a9da5fdef57dc4b,d02a171a7965aee6
Lung_Opacity-1624.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1624.png,Lung_Opacity/masks/Lung_Opacity-1624.png,True,G04097,test,a60935fd3e41ac3a7dfc41bd0178dff3,dc2a530f6c0e6eca
Lung_Opacity-1625.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1625.png,Lung_Opacity/masks/Lung_Opacity-1625.png,True,G04098,test,98e3b72706a86d7e577212950d4d8f5c,d42f0b5f4b0a0b9b
Lung_Opacity-1626.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1626.png,Lung_Opacity/masks/Lung_Opacity-1626.png,True,G04099,train,aec180aa4ce38282168c6764e8de85da,81285e1f5b7d4a5c
Lung_Opacity-1627.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1627.png,Lung_Opacity/masks/Lung_Opacity-1627.png,True,G04100,val,58ec6b279cc68215968e7d851c085216,d13be3c7961694c2
Lung_Opacity-1628.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1628.png,Lung_Opacity/masks/Lung_Opacity-1628.png,True,G04101,train,86bd05e502c12b569f26cc69ec803bab,917fc65b5b082b29
Lung_Opacity-1629.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1629.png,Lung_Opacity/masks/Lung_Opacity-1629.png,True,G04102,train,6711d5801004d54c1b5b066e6290cb1a,dc2e425f486c7ce8
Lung_Opacity-163.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-163.png,Lung_Opacity/masks/Lung_Opacity-163.png,True,G04103,val,004265872c3915c7094bf0ab2ca7d3bb,cc2a53414d5d1f2f
Lung_Opacity-1630.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1630.png,Lung_Opacity/masks/Lung_Opacity-1630.png,True,G04104,train,a8e3594e1e99b1d64254e5a5a269feaa,d52e4e0c2cb5f6a2
Lung_Opacity-1631.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1631.png,Lung_Opacity/masks/Lung_Opacity-1631.png,True,G04105,train,f14922e222b6efc21d08d845b195c01f,dc2c524b4a0b4f6f
Lung_Opacity-1632.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1632.png,Lung_Opacity/masks/Lung_Opacity-1632.png,True,G04106,train,90cd8ef403b6ef01f9222830c938d620,c12f160a4f4d4b6f
Lung_Opacity-1633.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1633.png,Lung_Opacity/masks/Lung_Opacity-1633.png,True,G04107,train,53ca5a8b12855171d8cbb8cb2f41e97e,dc28d30b45595d5b
Lung_Opacity-1634.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1634.png,Lung_Opacity/masks/Lung_Opacity-1634.png,True,G04108,test,f4e8e234f1f7e6c8176843b7a69ced20,d83c4b0f19cdcc3c
Lung_Opacity-1635.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1635.png,Lung_Opacity/masks/Lung_Opacity-1635.png,True,G04109,train,a7c5823becb3096c1ae3014a435e11fa,c17d9683121cdf4b
Lung_Opacity-1636.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1636.png,Lung_Opacity/masks/Lung_Opacity-1636.png,True,G04110,train,0cf9c93a2cd39e5c2ba7dd9529cb0afd,893d06061d5d5d5f
Lung_Opacity-1637.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1637.png,Lung_Opacity/masks/Lung_Opacity-1637.png,True,G04111,train,7034a5334fa0c6655ea564c8b2dc03a5,9173141e1a3b3f1e
Lung_Opacity-1638.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1638.png,Lung_Opacity/masks/Lung_Opacity-1638.png,True,G04112,train,7abcba12fe66f9aac9af63ddbfbb63fd,d96ac5945ec41f1a
Lung_Opacity-1639.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1639.png,Lung_Opacity/masks/Lung_Opacity-1639.png,True,G04113,train,43c287874cb1c44d89eb4b64419e46fb,d42d43830f6cf49e
Lung_Opacity-164.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-164.png,Lung_Opacity/masks/Lung_Opacity-164.png,True,G04114,train,0ffc9933f1934191f844195e7a53c269,817a5a4849373f37
Lung_Opacity-1640.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1640.png,Lung_Opacity/masks/Lung_Opacity-1640.png,True,G04115,train,88b1b00ce10a7d361f3a195704139528,817b961a596978e9
Lung_Opacity-1641.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1641.png,Lung_Opacity/masks/Lung_Opacity-1641.png,True,G04116,train,6cd9094c5a99f7d59d26c28fdc74be80,c97817161e1d8f39
Lung_Opacity-1642.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1642.png,Lung_Opacity/masks/Lung_Opacity-1642.png,True,G04117,val,e34b671cfe65f3c9dea20fcaf47416a7,81284e5f1e5a5f5a
Lung_Opacity-1643.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1643.png,Lung_Opacity/masks/Lung_Opacity-1643.png,True,G04118,test,e26177ec9337ef8940f56b8ffd129115,c07d56515b4f2c3c
Lung_Opacity-1644.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1644.png,Lung_Opacity/masks/Lung_Opacity-1644.png,True,G04119,test,49823b339253134a0d65a8e7b3819932,987f875a491d1947
Lung_Opacity-1645.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1645.png,Lung_Opacity/masks/Lung_Opacity-1645.png,True,G04120,train,d94ad174e07073dd105f3c8133e1709e,9169c61c1b15795f
Lung_Opacity-1646.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1646.png,Lung_Opacity/masks/Lung_Opacity-1646.png,True,G04121,train,ac26e7e87669fb9310ab5fb5fc67a059,dd2d461cc8496ce7
Lung_Opacity-1647.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1647.png,Lung_Opacity/masks/Lung_Opacity-1647.png,True,G04122,train,5bfcc78ad8a178cb1a355d121df934a2,d43d5a4b0e0e1d1f
Lung_Opacity-1648.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1648.png,Lung_Opacity/masks/Lung_Opacity-1648.png,True,G04123,train,18cba62f2b84c27beec0457a0d9c7905,ce35504a4a4f4f4f
Lung_Opacity-1649.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1649.png,Lung_Opacity/masks/Lung_Opacity-1649.png,True,G04124,train,7101102a363575d7a5ee05ab54557419,913f465a480b3b3f
Lung_Opacity-165.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-165.png,Lung_Opacity/masks/Lung_Opacity-165.png,True,G04125,train,4911130de5442e5e7531ddfeadb05439,dc6dc21e096d3c78
Lung_Opacity-1650.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1650.png,Lung_Opacity/masks/Lung_Opacity-1650.png,True,G04126,train,1559c004643f3a576aa06be53ca018b2,d93a86585b41ff81
Lung_Opacity-1651.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1651.png,Lung_Opacity/masks/Lung_Opacity-1651.png,True,G04127,train,dcc81cd04ee8724b0a6f2b86d889b182,933f8098a8ecacee
Lung_Opacity-1652.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1652.png,Lung_Opacity/masks/Lung_Opacity-1652.png,True,G04128,train,a9124b6b490faad8b7fab150f355c335,917a560a1f1b7b19
Lung_Opacity-1653.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1653.png,Lung_Opacity/masks/Lung_Opacity-1653.png,True,G04129,train,76ee34535b6cd35f16e0aadf897ff582,c42f074d2c6cede1
Lung_Opacity-1654.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1654.png,Lung_Opacity/masks/Lung_Opacity-1654.png,True,G04130,test,cec5e50f8076e215d4269e3ff77add29,9d3d421f49ca2d39
Lung_Opacity-1655.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1655.png,Lung_Opacity/masks/Lung_Opacity-1655.png,True,G04131,train,1dba37e49f726142f0db93fcc6c7910c,dc2c834b1dec6c78
Lung_Opacity-1656.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1656.png,Lung_Opacity/masks/Lung_Opacity-1656.png,True,G04132,train,dd85270685d0d70e074af5f96e228543,896d84da7ae56163
Lung_Opacity-1657.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1657.png,Lung_Opacity/masks/Lung_Opacity-1657.png,True,G04133,val,4100a71cc2d1cab0de957cb44c8a7609,913d46c6d218fcce
Lung_Opacity-1658.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1658.png,Lung_Opacity/masks/Lung_Opacity-1658.png,True,G04134,train,020fa724f3d488a1b70d199d519583fc,91615e1e9e1a1b5b
Lung_Opacity-1659.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1659.png,Lung_Opacity/masks/Lung_Opacity-1659.png,True,G04135,val,b346ce2b0d682f6147d036cb85d2a7fc,d42f0347454f666e
Lung_Opacity-166.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-166.png,Lung_Opacity/masks/Lung_Opacity-166.png,True,G04136,val,f1d28e321e81925bc47d2dbc0139e55d,d928d7930a0ecb5b
Lung_Opacity-1660.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1660.png,Lung_Opacity/masks/Lung_Opacity-1660.png,True,G04137,test,44ab23043cd35b242ecbfb6950607c61,817f0707161e5d4f
Lung_Opacity-1661.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1661.png,Lung_Opacity/masks/Lung_Opacity-1661.png,True,G04138,test,534c26b45841911ca9cb65d33e33a00b,d1bb525d4a293c4b
Lung_Opacity-1662.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1662.png,Lung_Opacity/masks/Lung_Opacity-1662.png,True,G04139,train,8defb2b8d4ada8098d673457a5f8391c,c0397b06da1d787a
Lung_Opacity-1663.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1663.png,Lung_Opacity/masks/Lung_Opacity-1663.png,True,G04140,train,0bb985ce8c0ffd0dd0d3cf0ee1f91657,d4b5cf0d0c2c253f
Lung_Opacity-1664.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1664.png,Lung_Opacity/masks/Lung_Opacity-1664.png,True,G04141,train,8cc4b2554eadea757e3a5b1a6b8689c1,d42ae703554e5d2e
Lung_Opacity-1665.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1665.png,Lung_Opacity/masks/Lung_Opacity-1665.png,True,G04142,val,5ca98db0af3e83b37411303c5a85ddc0,9129c61e1a8b3fb9
Lung_Opacity-1666.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1666.png,Lung_Opacity/masks/Lung_Opacity-1666.png,True,G04143,test,de6549d56ba1b8606fb2f0aa0062a51c,d96b061a69313e67
Lung_Opacity-1667.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1667.png,Lung_Opacity/masks/Lung_Opacity-1667.png,True,G04144,test,5276c98a65edfbf51b66ad3342c8c9d3,cc3a5b4f0e293c2d
Lung_Opacity-1668.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1668.png,Lung_Opacity/masks/Lung_Opacity-1668.png,True,G04145,train,21f299d29ad66a66a2de48d8272765a0,812c878b8ba9b1fb
Lung_Opacity-1669.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1669.png,Lung_Opacity/masks/Lung_Opacity-1669.png,True,G04146,train,9768787660df7e18d8a2a15554fac495,c43f5f482d2c076e
Lung_Opacity-167.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-167.png,Lung_Opacity/masks/Lung_Opacity-167.png,True,G04147,train,13c1a5f231a333bd71279d8f819611b4,813a561e1a5b1b3f
Lung_Opacity-1670.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1670.png,Lung_Opacity/masks/Lung_Opacity-1670.png,True,G04148,train,017d6da3b9ae463c1299a87fa3085b49,d02f870c3d2f0d4f
Lung_Opacity-1671.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1671.png,Lung_Opacity/masks/Lung_Opacity-1671.png,True,G04149,train,e856556b81354f1014110e6e27202796,cc7f430f0b2d2f24
Lung_Opacity-1672.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1672.png,Lung_Opacity/masks/Lung_Opacity-1672.png,True,G04150,train,3c5b1cd351766cdd958d23e87558b1d6,956a275762a73a91
Lung_Opacity-1673.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1673.png,Lung_Opacity/masks/Lung_Opacity-1673.png,True,G04151,train,3ba9c9b869574195e0a702a78da00f21,8128165e5b7b7ae8
Lung_Opacity-1674.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1674.png,Lung_Opacity/masks/Lung_Opacity-1674.png,True,G04152,test,c746929134715b94e4fa18d9d3df54cf,dd2e50530b4bed4a
Lung_Opacity-1675.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1675.png,Lung_Opacity/masks/Lung_Opacity-1675.png,True,G04153,train,5fc2b9737ad24133ffbb972de8885269,890f075e6b6d6961
Lung_Opacity-1676.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1676.png,Lung_Opacity/masks/Lung_Opacity-1676.png,True,G04154,train,4925e39a0b39584375b063398ceb6b73,952e871f0d0e2e6e
Lung_Opacity-1677.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1677.png,Lung_Opacity/masks/Lung_Opacity-1677.png,True,G04155,train,43abd8eb75e07bc747f2636c30b4d3e9,9b7f1c131b3b1852
Lung_Opacity-1678.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1678.png,Lung_Opacity/masks/Lung_Opacity-1678.png,True,G04156,train,359dd852543ea6e0638cd75f3f727d20,83781e121d7b73c3
Lung_Opacity-1679.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1679.png,Lung_Opacity/masks/Lung_Opacity-1679.png,True,G04157,val,3cc44c9306fbf129fb1db2efb8a257b9,817bc6961a5b5d19
Lung_Opacity-168.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-168.png,Lung_Opacity/masks/Lung_Opacity-168.png,True,G04158,val,3e85b7093685592f0cde393f87d4d343,887b570e171f1a1e
Lung_Opacity-1680.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1680.png,Lung_Opacity/masks/Lung_Opacity-1680.png,True,G04159,train,5e7d1723a779ebc93cbe9416d1cd06be,c43d134d0eea3d9c
Lung_Opacity-1681.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1681.png,Lung_Opacity/masks/Lung_Opacity-1681.png,True,G04160,train,e31267edc159b1b50bf79024dd14d40f,93681c5b58395bd5
Lung_Opacity-1682.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1682.png,Lung_Opacity/masks/Lung_Opacity-1682.png,True,G04161,train,567c81b6dfcd55ed6b3f4b490218094b,917bc3173a4d3938
Lung_Opacity-1683.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1683.png,Lung_Opacity/masks/Lung_Opacity-1683.png,True,G04162,train,843530abfeadae3638232bf7ef84fe45,de36430a2f262f0f
Lung_Opacity-1684.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1684.png,Lung_Opacity/masks/Lung_Opacity-1684.png,True,G04163,train,05d5c65da515fdb0c85037181e62a4c3,d129565a0e1d2f2f
Lung_Opacity-1685.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1685.png,Lung_Opacity/masks/Lung_Opacity-1685.png,True,G04164,train,e56f645afdb5838bbb322d58e6e9b552,807b5a5d0d5b5a39
Lung_Opacity-1686.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1686.png,Lung_Opacity/masks/Lung_Opacity-1686.png,True,G04165,train,76d27aed44558b41d6ce3000410913df,dd6c071b08653eda
Lung_Opacity-1687.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1687.png,Lung_Opacity/masks/Lung_Opacity-1687.png,True,G04166,val,2e6fbdb1140d1bead0298436b5ae7781,d42e430b4b4e2e6f
Lung_Opacity-1688.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1688.png,Lung_Opacity/masks/Lung_Opacity-1688.png,True,G04167,train,a20ab1749a076e63d491dbbc84b7ce18,c13e165e5d7c7848
Lung_Opacity-1689.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1689.png,Lung_Opacity/masks/Lung_Opacity-1689.png,True,G04168,train,6f722d2b7c7b38a038bc8cd015cdcc09,c83d16525b5d1e1e
Lung_Opacity-169.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-169.png,Lung_Opacity/masks/Lung_Opacity-169.png,True,G04169,train,bb3b3a1b33c0b244519960c9b58e7586,856a60c4c6d65b7b
Lung_Opacity-1690.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1690.png,Lung_Opacity/masks/Lung_Opacity-1690.png,True,G04170,train,4b56c46ad8b6d366956da7ce7d8b78b6,906bd716495c5d5c
Lung_Opacity-1691.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1691.png,Lung_Opacity/masks/Lung_Opacity-1691.png,True,G04171,train,f971d174b83459fb239c484390b05f3c,d46d430f0b6c3c7c
Lung_Opacity-1692.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1692.png,Lung_Opacity/masks/Lung_Opacity-1692.png,True,G04172,train,3d837b6cd2d638464e3897608bb4015a,8d2c3cf2c2c79746
Lung_Opacity-1693.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1693.png,Lung_Opacity/masks/Lung_Opacity-1693.png,True,G04173,val,abfd16b7d801147fcf18c26cad92900e,c13f561654535b5a
Lung_Opacity-1694.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1694.png,Lung_Opacity/masks/Lung_Opacity-1694.png,True,G04174,train,ee6201a9994f604322645c95d07f0e09,c455530b6f2f0e2d
Lung_Opacity-1695.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1695.png,Lung_Opacity/masks/Lung_Opacity-1695.png,True,G04175,test,d5dec434442333d5173baedb2914bb6e,897c47821b4f0f1f
Lung_Opacity-1696.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1696.png,Lung_Opacity/masks/Lung_Opacity-1696.png,True,G04176,test,4274359121ce114075b03078daf2b143,d17bd616145b5b50
Lung_Opacity-1697.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1697.png,Lung_Opacity/masks/Lung_Opacity-1697.png,True,G04177,test,79b7378455c47b5639e82f794c5af9b4,817b141e595b1b5b
Lung_Opacity-1698.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1698.png,Lung_Opacity/masks/Lung_Opacity-1698.png,True,G04178,val,07901b186f403e0a45801bfbb72813eb,d43c93528f5a9b4a
Lung_Opacity-1699.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1699.png,Lung_Opacity/masks/Lung_Opacity-1699.png,True,G04179,train,c000b7a35d509b5574249817420b39ee,d43d07d70a0c6f4e
Lung_Opacity-17.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-17.png,Lung_Opacity/masks/Lung_Opacity-17.png,True,G04180,train,3400cb0465e3a8fbe2bb8468309e30a6,cc6e534b4d0e2e2d
Lung_Opacity-170.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-170.png,Lung_Opacity/masks/Lung_Opacity-170.png,True,G04181,train,bc1fae5c914a5c6f0932c36a4677537f,917b021e1b5b1b1f
Lung_Opacity-1700.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1700.png,Lung_Opacity/masks/Lung_Opacity-1700.png,True,G04182,train,985d9bbcf900230eab8168fd99599d31,dc2e430d0c2cedfa
Lung_Opacity-1701.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1701.png,Lung_Opacity/masks/Lung_Opacity-1701.png,True,G04183,train,e2b5ef7dd515495258a22fb097a6c464,917b4616185f4f39
Lung_Opacity-1702.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1702.png,Lung_Opacity/masks/Lung_Opacity-1702.png,True,G04184,train,76556b4099b3628bddf85a3c8e813799,896c461b4d38adfc
Lung_Opacity-1703.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1703.png,Lung_Opacity/masks/Lung_Opacity-1703.png,True,G04185,train,e45170b540ca7359c03155e41e2bac86,8c68ae07fad8e9e0
Lung_Opacity-1704.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1704.png,Lung_Opacity/masks/Lung_Opacity-1704.png,True,G04186,train,9dc33c322fd033d3b5c2c247826253c8,c52f56485d0f5d15
Lung_Opacity-1705.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1705.png,Lung_Opacity/masks/Lung_Opacity-1705.png,True,G04187,val,e1335cc3435dd7e576c7506378930906,807f0657125f5b59
Lung_Opacity-1706.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1706.png,Lung_Opacity/masks/Lung_Opacity-1706.png,True,G04188,train,4a1c7289a96e1c34b6ad681dc070e955,dd7ad34b5c68190a
Lung_Opacity-1707.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1707.png,Lung_Opacity/masks/Lung_Opacity-1707.png,True,G04189,train,1612a698b398c77a7feb13c4ebe9875e,c839d208292b3fbf
Lung_Opacity-1708.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1708.png,Lung_Opacity/masks/Lung_Opacity-1708.png,True,G04190,test,9f2ab465085d8026648798e121c4f62f,943f41431a5f4f0f
Lung_Opacity-1709.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1709.png,Lung_Opacity/masks/Lung_Opacity-1709.png,True,G04191,train,33aa5e06e638ed735b84dfd8ef1443d1,946ea31d4c3c6c4f
Lung_Opacity-171.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-171.png,Lung_Opacity/masks/Lung_Opacity-171.png,True,G04192,train,425799a794b45eb50b80e034fbae7c91,d13b561e5d491d19
Lung_Opacity-1710.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1710.png,Lung_Opacity/masks/Lung_Opacity-1710.png,True,G04193,test,c48330c63e423ec32ce8a9039d43dc2c,887f05465e5a7965
Lung_Opacity-1711.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1711.png,Lung_Opacity/masks/Lung_Opacity-1711.png,True,G04194,train,ce570ea4ed05e9f4349988d7e25c3800,dc2d430b4e2e0f9b
Lung_Opacity-1712.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1712.png,Lung_Opacity/masks/Lung_Opacity-1712.png,True,G04195,train,3149983391a6b1f2deeb2ac2302242a9,893f061a194d5fd9
Lung_Opacity-1713.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1713.png,Lung_Opacity/masks/Lung_Opacity-1713.png,True,G04196,val,d7550cbad05a3cd8be4bf7fe3d4b99c7,9179c61c124f7979
Lung_Opacity-1714.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1714.png,Lung_Opacity/masks/Lung_Opacity-1714.png,True,G04197,val,f899c4d0720c1972ec67e9d0199a9c4d,c42ac14d0e3d2f3f
Lung_Opacity-1715.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1715.png,Lung_Opacity/masks/Lung_Opacity-1715.png,True,G04198,test,3fbcd059aea71799c9e9994947894d29,d42e410f6e623bf8
Lung_Opacity-1716.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1716.png,Lung_Opacity/masks/Lung_Opacity-1716.png,True,G04199,train,8ac01cbe0c28b965933473d3a13003d1,cd2e524b4d0e0f1f
Lung_Opacity-1717.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1717.png,Lung_Opacity/masks/Lung_Opacity-1717.png,True,G04200,test,d7fa1cea03c1bddcdbfdf718edc27220,d07d431a1b4b3d5c
Lung_Opacity-1718.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1718.png,Lung_Opacity/masks/Lung_Opacity-1718.png,True,G04201,train,3b9cd39b7736b166118294e069417041,816f4f4a4e4b4b4b
Lung_Opacity-1719.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1719.png,Lung_Opacity/masks/Lung_Opacity-1719.png,True,G04202,train,f2cb579514206d1f0466703b70794a30,887bc61a5b493979
Lung_Opacity-172.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-172.png,Lung_Opacity/masks/Lung_Opacity-172.png,True,G04203,train,fb7e4b692274ca285674f2fe9bc9e33f,d528554f0a4f0d3f
Lung_Opacity-1720.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1720.png,Lung_Opacity/masks/Lung_Opacity-1720.png,True,G04204,train,d2ee1cfd8c74f71cbfcfd4f7d28b0e03,c00d4f0a2e2f3f6d
Lung_Opacity-1721.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1721.png,Lung_Opacity/masks/Lung_Opacity-1721.png,True,G04205,train,ab40a7168975db6d52c35be71379c6e6,997b441e524b5d5c
Lung_Opacity-1722.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1722.png,Lung_Opacity/masks/Lung_Opacity-1722.png,True,G04206,train,21b851c57e9b996a7e894cf92baf7607,9839571e1879cba9
Lung_Opacity-1723.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1723.png,Lung_Opacity/masks/Lung_Opacity-1723.png,True,G04207,train,6c3c25bc87e6e2fcb84f071d67b154a7,c02d7e5ed656d242
Lung_Opacity-1724.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1724.png,Lung_Opacity/masks/Lung_Opacity-1724.png,True,G04208,train,37d72c8321d94e3052bc962f2461b900,c82a4f4a0e2d7d1f
Lung_Opacity-1725.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1725.png,Lung_Opacity/masks/Lung_Opacity-1725.png,True,G04209,train,52f7f5618ae475b2208029ab687a1119,917e0618597979e9
Lung_Opacity-1726.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1726.png,Lung_Opacity/masks/Lung_Opacity-1726.png,True,G04210,train,58ef60613fd09f96e5cf2ba824db9097,c626616a3a3d9d1d
Lung_Opacity-1727.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1727.png,Lung_Opacity/masks/Lung_Opacity-1727.png,True,G04211,train,eb2066b948cd41462db1a6cd1fbdabb3,dd2e40c3174d5a7c
Lung_Opacity-1728.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1728.png,Lung_Opacity/masks/Lung_Opacity-1728.png,True,G04212,train,f05d2d7de7afd680abb639be993d66ee,9139c31e1b497c3e
Lung_Opacity-1729.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1729.png,Lung_Opacity/masks/Lung_Opacity-1729.png,True,G04213,train,c7c2091ee7b91b04aa3ed8965a0c609f,d42d530a2c2c3bf7
Lung_Opacity-173.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-173.png,Lung_Opacity/masks/Lung_Opacity-173.png,True,G04214,test,c6ca90e1abd83ca5e1c2c55d38f75567,ddae431f431d114d
Lung_Opacity-1730.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1730.png,Lung_Opacity/masks/Lung_Opacity-1730.png,True,G04215,test,f5bdc53b88089a0b67b7b68076865c1d,dc2ed2530f4c4e1b
Lung_Opacity-1731.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1731.png,Lung_Opacity/masks/Lung_Opacity-1731.png,True,G04216,val,c881024eb479c0cf411e73a1e19dc111,c42f434e4c6ee6e1
Lung_Opacity-1732.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1732.png,Lung_Opacity/masks/Lung_Opacity-1732.png,True,G04217,val,30ff2d6bb5c426b7007442e8240c117f,943c45c75b59485f
Lung_Opacity-1733.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1733.png,Lung_Opacity/masks/Lung_Opacity-1733.png,True,G04218,val,1a13d52c65c1bc371930f89af904f8f6,c53e064e5c1c4b5f
Lung_Opacity-1734.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1734.png,Lung_Opacity/masks/Lung_Opacity-1734.png,True,G04219,train,b24a085fdedffe5986e963322af2abd5,d03dda1b9e0cb83c
Lung_Opacity-1735.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1735.png,Lung_Opacity/masks/Lung_Opacity-1735.png,True,G04220,train,0ef4c304635e9fce9e8a0c2f798bbfb1,817f861e5969f941
Lung_Opacity-1736.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1736.png,Lung_Opacity/masks/Lung_Opacity-1736.png,True,G04221,train,70466d6ec2c5f4d865cd20071799d984,817a3a49195d7979
Lung_Opacity-1737.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1737.png,Lung_Opacity/masks/Lung_Opacity-1737.png,True,G04222,train,3738637d04799503b5aec1ad6750fc91,81789e191f5a3a5b
Lung_Opacity-1738.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1738.png,Lung_Opacity/masks/Lung_Opacity-1738.png,True,G04223,train,fc15948332021f77490584df1fb1376f,9528075e1c5d4d7e
Lung_Opacity-1739.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1739.png,Lung_Opacity/masks/Lung_Opacity-1739.png,True,G04224,train,14128e0f111518c67c3d5a8d10b11066,cc2d430f0f4d6d2d
Lung_Opacity-174.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-174.png,Lung_Opacity/masks/Lung_Opacity-174.png,True,G04225,train,e1d4b7c466486b99bc855c39e6c49177,897d161a59797949
Lung_Opacity-1740.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1740.png,Lung_Opacity/masks/Lung_Opacity-1740.png,True,G04226,val,21153b314a2063c70125b0639209b2d4,dc3d5b435f68482c
Lung_Opacity-1741.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1741.png,Lung_Opacity/masks/Lung_Opacity-1741.png,True,G04227,val,0df184beb540a8ffd28509d587f77337,814f021fd8fef0f0
Lung_Opacity-1742.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1742.png,Lung_Opacity/masks/Lung_Opacity-1742.png,True,G04228,train,55e08ec200c8766036c2511bd28b2249,91395e3b3a35b0b4
Lung_Opacity-1743.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1743.png,Lung_Opacity/masks/Lung_Opacity-1743.png,True,G04229,train,242cb3665d14d724bffe2832f4643bf9,886d064a497dd5fa
Lung_Opacity-1744.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1744.png,Lung_Opacity/masks/Lung_Opacity-1744.png,True,G04230,train,3cc2e43a0307ca882f7c61b3190b8a4e,c02d121b4a3d3f7d
Lung_Opacity-1745.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1745.png,Lung_Opacity/masks/Lung_Opacity-1745.png,True,G04231,val,8887889de60af6dcfe438e7d971dcd65,9178204a5f5f1b5b
Lung_Opacity-1746.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1746.png,Lung_Opacity/masks/Lung_Opacity-1746.png,True,G04232,test,92308ffcd07f907031dab46c12f50419,d57b52830f3d2934
Lung_Opacity-1747.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1747.png,Lung_Opacity/masks/Lung_Opacity-1747.png,True,G04233,test,054e2868371bbe0973c36409ab8173f7,d46941175a555f59
Lung_Opacity-1748.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1748.png,Lung_Opacity/masks/Lung_Opacity-1748.png,True,G04234,train,5733460a7699922fc4ae48d9e6a6f8ea,dd6b06961bc93ac4
Lung_Opacity-1749.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1749.png,Lung_Opacity/masks/Lung_Opacity-1749.png,True,G04235,train,4b26611d4c9532541c3db5d63a33fc1b,896f06961a1ccf1f
Lung_Opacity-175.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-175.png,Lung_Opacity/masks/Lung_Opacity-175.png,True,G04236,train,f148d46f588afe59ff89bb142610e28e,9c0c4b4d2c6ceef2
Lung_Opacity-1750.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1750.png,Lung_Opacity/masks/Lung_Opacity-1750.png,True,G04237,train,fdb0ad574a35825ef87af10ecb4f951b,c02dbe53cb11acec
Lung_Opacity-1751.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1751.png,Lung_Opacity/masks/Lung_Opacity-1751.png,True,G04238,train,c5a52dc9ca1087e21e610ba719f9f165,dd2e428b0fc9ba1c
Lung_Opacity-1752.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1752.png,Lung_Opacity/masks/Lung_Opacity-1752.png,True,G04239,test,02bd4f4e8c76845f407315f7fca8def9,dd2a5756525b5858
Lung_Opacity-1753.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1753.png,Lung_Opacity/masks/Lung_Opacity-1753.png,True,G04240,train,4b361189b7eacaf4b9edcbba1915c457,8b781417175a5b5b
Lung_Opacity-1754.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1754.png,Lung_Opacity/masks/Lung_Opacity-1754.png,True,G04241,train,04f99af94a7341e4f099eed01704b4b7,c02b7f024a6b6c7b
Lung_Opacity-1755.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1755.png,Lung_Opacity/masks/Lung_Opacity-1755.png,True,G04242,train,9338cb0681f6dbf2550be625ba1c8d8b,986fc6184f9c4a9d
Lung_Opacity-1756.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1756.png,Lung_Opacity/masks/Lung_Opacity-1756.png,True,G04243,train,e2be86d50e883fe1975861a3f8265878,c3d727c127e47078
Lung_Opacity-1757.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1757.png,Lung_Opacity/masks/Lung_Opacity-1757.png,True,G04244,train,e6f4e69045ed676ce41fb16f308ccd1d,d52a474f591d083f
Lung_Opacity-1758.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1758.png,Lung_Opacity/masks/Lung_Opacity-1758.png,True,G04245,train,7349a989aff4388182e40b0f11c676f1,912bc6184d5b3b5d
Lung_Opacity-1759.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1759.png,Lung_Opacity/masks/Lung_Opacity-1759.png,True,G04246,train,a0c07724597e132f8148089c4d97e90b,8178101b1b5b7b7e
Lung_Opacity-176.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-176.png,Lung_Opacity/masks/Lung_Opacity-176.png,True,G04247,train,7b9130e58d22e3286db898fc27fcbcc1,852e0f1a494f4f4f
Lung_Opacity-1760.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1760.png,Lung_Opacity/masks/Lung_Opacity-1760.png,True,G04248,train,a98f69b68165af6301fd24f4706c821a,817f16185d5d4a7a
Lung_Opacity-1761.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1761.png,Lung_Opacity/masks/Lung_Opacity-1761.png,True,G04249,train,b73c2865c650d9fa07beb05b3abca6e6,803b58195d5efe2c
Lung_Opacity-1762.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1762.png,Lung_Opacity/masks/Lung_Opacity-1762.png,True,G04250,train,85c8d2be33f77883acc8e3129c1080c9,cd2e12191c4e6bf5
Lung_Opacity-1763.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1763.png,Lung_Opacity/masks/Lung_Opacity-1763.png,True,G04251,train,caf4981883bbda76be39ba0a538cc866,c94d05574f4b4d4d
Lung_Opacity-1764.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1764.png,Lung_Opacity/masks/Lung_Opacity-1764.png,True,G04252,train,c48518f61ab5fc902c55d9a06c716f3b,d0315bd3570e4f4a
Lung_Opacity-1765.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1765.png,Lung_Opacity/masks/Lung_Opacity-1765.png,True,G04253,train,62fdd197abd8c38b00910348d5a992ab,d42d430b1f4b0f2f
Lung_Opacity-1766.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1766.png,Lung_Opacity/masks/Lung_Opacity-1766.png,True,G04254,val,61e8f5092dfd33e6a623f8ce4358a662,c44f074d486a6afb
Lung_Opacity-1767.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1767.png,Lung_Opacity/masks/Lung_Opacity-1767.png,True,G04255,val,850eae4f0789747159c1ac62388a3a87,d12b5c507b4b7929
Lung_Opacity-1768.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1768.png,Lung_Opacity/masks/Lung_Opacity-1768.png,True,G04256,val,a5e631b84dc874cd6ef19ee5030dd55e,d03d07474b793c56
Lung_Opacity-1769.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1769.png,Lung_Opacity/masks/Lung_Opacity-1769.png,True,G04257,test,aa1e76effea00cae6663875659d57465,93781f7b4a4ec1c4
Lung_Opacity-177.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-177.png,Lung_Opacity/masks/Lung_Opacity-177.png,True,G04258,test,3c7d10842621942ee315b6e2f56b4b9b,897c165a595a69ec
Lung_Opacity-1770.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1770.png,Lung_Opacity/masks/Lung_Opacity-1770.png,True,G04259,train,ae984706a11fb88df72836684672171a,d52f404a4ee96c7c
Lung_Opacity-1771.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1771.png,Lung_Opacity/masks/Lung_Opacity-1771.png,True,G04260,test,4093f73509105b0d569781f0bcb27e3e,917bc617070f1a1f
Lung_Opacity-1772.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1772.png,Lung_Opacity/masks/Lung_Opacity-1772.png,True,G04261,train,a70ca647d2e6e1e521c1a3d3496e280b,d42f53d94c3c382b
Lung_Opacity-1773.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1773.png,Lung_Opacity/masks/Lung_Opacity-1773.png,True,G04262,train,0cf6f559e853ebff6689a9931ddba421,c87b3586561a395b
Lung_Opacity-1774.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1774.png,Lung_Opacity/masks/Lung_Opacity-1774.png,True,G04263,train,7612f436b59cc992c787be7dba7855a2,c43e434b4f6d6564
Lung_Opacity-1775.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1775.png,Lung_Opacity/masks/Lung_Opacity-1775.png,True,G04264,train,f0a233bc44c3f5240fe945b10988fd66,883f075e18494dfd
Lung_Opacity-1776.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1776.png,Lung_Opacity/masks/Lung_Opacity-1776.png,True,G04265,train,11cefdec77fcbb1677f6d8dfdcf89d9f,95626965ccadb392
Lung_Opacity-1777.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1777.png,Lung_Opacity/masks/Lung_Opacity-1777.png,True,G04266,train,5f5991204e9317eb2e4f00e27706cf27,9179465e1e5a5a5a
Lung_Opacity-1778.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1778.png,Lung_Opacity/masks/Lung_Opacity-1778.png,True,G04267,train,79b0f3fe9defec384c8296d670c2a1a9,917a061a5b7959f8
Lung_Opacity-1779.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1779.png,Lung_Opacity/masks/Lung_Opacity-1779.png,True,G04268,train,e94efbf23cb6cc366dfdb15339feaa7c,946aa783831d1f1f
Lung_Opacity-178.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-178.png,Lung_Opacity/masks/Lung_Opacity-178.png,True,G04269,train,c9fd0fe0bc8268773bec91c3012a5223,d03d17c20a0f6f2f
Lung_Opacity-1780.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1780.png,Lung_Opacity/masks/Lung_Opacity-1780.png,True,G04270,train,7ded0b82f18ef9145e423d24cfa63b12,dd27024b0c68e5f7
Lung_Opacity-1781.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1781.png,Lung_Opacity/masks/Lung_Opacity-1781.png,True,G04271,val,c5555a9d3a2b3568d97a3d87e6b897db,90296fc63ed42fc1
Lung_Opacity-1782.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1782.png,Lung_Opacity/masks/Lung_Opacity-1782.png,True,G04272,val,7c47e22c4c31e7fd506d8da1e9728f79,dc2d431e68727273
Lung_Opacity-1783.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1783.png,Lung_Opacity/masks/Lung_Opacity-1783.png,True,G04273,train,22728c788a79322db2ae61f2cb533572,947e4b475f4c4c4c
Lung_Opacity-1784.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1784.png,Lung_Opacity/masks/Lung_Opacity-1784.png,True,G04274,train,f540eded6bab8e91f86b18908999edb5,8131525f7a3b7a38
Lung_Opacity-1785.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1785.png,Lung_Opacity/masks/Lung_Opacity-1785.png,True,G04275,train,6ee54e788e47763af3a851092628f849,8b34d7d0f0e30f0d
Lung_Opacity-1786.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1786.png,Lung_Opacity/masks/Lung_Opacity-1786.png,True,G04276,val,c44b1778b062b2bff4817dafa66c34c0,cd2d92930f0d0f1f
Lung_Opacity-1787.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1787.png,Lung_Opacity/masks/Lung_Opacity-1787.png,True,G04277,test,374fd8f3402aa404d5d67a5dcc536aa1,811d161e5e785b7a
Lung_Opacity-1788.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1788.png,Lung_Opacity/masks/Lung_Opacity-1788.png,True,G04278,test,e0e56daeabe8fa46506390ab419bd178,c82d4b0f2e2aaaaf
Lung_Opacity-1789.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1789.png,Lung_Opacity/masks/Lung_Opacity-1789.png,True,G04279,train,9fc0e36db890a6f528b54a97ca89569b,da2d595b0b2f1616
Lung_Opacity-179.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-179.png,Lung_Opacity/masks/Lung_Opacity-179.png,True,G04280,test,02b7e123fe5561f3e23e97334f7b8afe,807bc71e1b4f1b0b
Lung_Opacity-1790.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1790.png,Lung_Opacity/masks/Lung_Opacity-1790.png,True,G04281,train,92248dda1fe9ab18f6f5c58239ad0156,917b87141e5b1b5a
Lung_Opacity-1791.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1791.png,Lung_Opacity/masks/Lung_Opacity-1791.png,True,G04282,train,6fbb47e5d8bc4c419d0f104bda6743c5,817d1f96921b0f1e
Lung_Opacity-1792.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1792.png,Lung_Opacity/masks/Lung_Opacity-1792.png,True,G04283,train,649c79936b03373c0c36f189bfcfb3f9,c92916165b4d5b5b
Lung_Opacity-1793.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1793.png,Lung_Opacity/masks/Lung_Opacity-1793.png,True,G04284,train,ce93acb1a283854e7e1f0c25a47b111c,85787f6281951e3b
Lung_Opacity-1794.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1794.png,Lung_Opacity/masks/Lung_Opacity-1794.png,True,G04285,train,23c20b1e2e460b58cc4e51b4df14cb87,d43d070e2d3e2c6d
Lung_Opacity-1795.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1795.png,Lung_Opacity/masks/Lung_Opacity-1795.png,True,G04286,train,58ae59d5ed98b96d682db25a8704d33f,d52d429e1b4e5a5a
Lung_Opacity-1796.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1796.png,Lung_Opacity/masks/Lung_Opacity-1796.png,True,G04287,train,7a03a036eee1791f92d9ded833131ef3,d6350f0e4c0f6e6c
Lung_Opacity-1797.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1797.png,Lung_Opacity/masks/Lung_Opacity-1797.png,True,G04288,train,6116f49ae02ecc87f0261d9a5890ff55,81780dd4525b5b5f
Lung_Opacity-1798.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1798.png,Lung_Opacity/masks/Lung_Opacity-1798.png,True,G04289,train,4031e9dc5cb12a0979451d8e3b717939,913b52101a3f3b3f
Lung_Opacity-1799.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1799.png,Lung_Opacity/masks/Lung_Opacity-1799.png,True,G04290,train,b34ce872466bb46be7dbd0f4f8e30fe2,8d685d5e486e1ea6
Lung_Opacity-18.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-18.png,Lung_Opacity/masks/Lung_Opacity-18.png,True,G04291,train,c13bfb924f8f9f2131b3eba930d065b1,d47e03530b4b0f1f
Lung_Opacity-180.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-180.png,Lung_Opacity/masks/Lung_Opacity-180.png,True,G04292,val,e619945bc83e1d6ea4a4535f6a02b61b,887945c6d217d74b
Lung_Opacity-1800.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1800.png,Lung_Opacity/masks/Lung_Opacity-1800.png,True,G04293,val,0dce453d740c6871d7daa726b53d3013,c52e824b0f0f2f6e
Lung_Opacity-1801.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1801.png,Lung_Opacity/masks/Lung_Opacity-1801.png,True,G04294,test,2ae32b1387c0c4580cb42b83453af0fe,d52a4b4d0e0e8b9f
Lung_Opacity-1802.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1802.png,Lung_Opacity/masks/Lung_Opacity-1802.png,True,G04295,test,a5d3f2503c10bd6629da9301459f2fe0,886f065e4fe9e1e0
Lung_Opacity-1803.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1803.png,Lung_Opacity/masks/Lung_Opacity-1803.png,True,G04296,train,dc3b41d19cdc05fe5b54531efe856841,db7fd4145f131302
Lung_Opacity-1804.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1804.png,Lung_Opacity/masks/Lung_Opacity-1804.png,True,G04297,train,2523ded0f481c46cb20a767ad7ccd067,8d2e838b9578bd49
Lung_Opacity-1805.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1805.png,Lung_Opacity/masks/Lung_Opacity-1805.png,True,G04298,test,7964b2074ff9a433b09a524c014db6ee,d41a674f5362926b
Lung_Opacity-1806.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1806.png,Lung_Opacity/masks/Lung_Opacity-1806.png,True,G04299,val,17512e36d650d2992ccc8806f60452af,d92d461b2d283d6d
Lung_Opacity-1807.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1807.png,Lung_Opacity/masks/Lung_Opacity-1807.png,True,G04300,train,c83b7015473f8afe8a65ac262175ba89,d13ad25a5c683b1d
Lung_Opacity-1808.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1808.png,Lung_Opacity/masks/Lung_Opacity-1808.png,True,G04301,train,4a17b47a035fa4ed4ec0f406a1cc3d5f,d42f07181d5d5d5c
Lung_Opacity-1809.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1809.png,Lung_Opacity/masks/Lung_Opacity-1809.png,True,G04302,train,d4960326b66f025396d1120334d78dec,996da51ac116baa7
Lung_Opacity-181.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-181.png,Lung_Opacity/masks/Lung_Opacity-181.png,True,G04303,train,69b40b65ee71faa6b832cda045ce9ad9,817e061a596d76f2
Lung_Opacity-1810.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1810.png,Lung_Opacity/masks/Lung_Opacity-1810.png,True,G04304,train,dfd3cc7c29843ac5e9ece29f561e30ba,cd3c12534b6d0c3f
Lung_Opacity-1811.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1811.png,Lung_Opacity/masks/Lung_Opacity-1811.png,True,G04305,train,ba425054865ebc1e724d72d7e251c088,d629404f2a2dbe5e
Lung_Opacity-1812.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1812.png,Lung_Opacity/masks/Lung_Opacity-1812.png,True,G04306,train,76899a4f81203c0cc4039101b9015d65,916e061e69797c4e
Lung_Opacity-1813.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1813.png,Lung_Opacity/masks/Lung_Opacity-1813.png,True,G04307,train,11fba6283b0812d83dbb9e4571038eb9,d528e0c397d2d353
Lung_Opacity-1814.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1814.png,Lung_Opacity/masks/Lung_Opacity-1814.png,True,G04308,train,f42f9726c0d277db4dc84f68932a7cc9,d42ad14f0e5a2f0f
Lung_Opacity-1815.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1815.png,Lung_Opacity/masks/Lung_Opacity-1815.png,True,G04309,train,70a1f108dc3fbfd31c2dfecde9c859a1,817f564a5b5b0a0f
Lung_Opacity-1816.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1816.png,Lung_Opacity/masks/Lung_Opacity-1816.png,True,G04310,train,7cbb67ad4a39cf20081431345ecd71fe,993dc04b1f9dc81b
Lung_Opacity-1817.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1817.png,Lung_Opacity/masks/Lung_Opacity-1817.png,True,G04311,val,18695222f2c1e9f581e84471edc0988a,807f8a12195d3f1f
Lung_Opacity-1818.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1818.png,Lung_Opacity/masks/Lung_Opacity-1818.png,True,G04312,train,90e29c08cef380f2c07f455a45f85e04,89780617161f5f1f
Lung_Opacity-1819.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1819.png,Lung_Opacity/masks/Lung_Opacity-1819.png,True,G04313,train,514d5104ac501080d341be63f609b870,98574e5b1a1b0f1e
Lung_Opacity-182.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-182.png,Lung_Opacity/masks/Lung_Opacity-182.png,True,G04314,val,5c2c8915946102ae1da70ac6e144665c,d42f1f1f1d0d1c1c
Lung_Opacity-1820.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1820.png,Lung_Opacity/masks/Lung_Opacity-1820.png,True,G04315,train,d86d5eb36410360a0c16191a8d3fac38,c47d48090bbdd7f0
Lung_Opacity-1821.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1821.png,Lung_Opacity/masks/Lung_Opacity-1821.png,True,G04316,test,c92cb130cad20101e760ce480298f82e,983dc5561e464f1b
Lung_Opacity-1822.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1822.png,Lung_Opacity/masks/Lung_Opacity-1822.png,True,G04317,val,b1262c02f1385b604620331be96ef9dc,d52e9051d51d0f4f
Lung_Opacity-1823.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1823.png,Lung_Opacity/masks/Lung_Opacity-1823.png,True,G04318,test,40cec64bde97851d2ee981a50c225505,8928171e1f5e787c
Lung_Opacity-1824.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1824.png,Lung_Opacity/masks/Lung_Opacity-1824.png,True,G04319,test,ac7b328b7d361a3b60bff32834d09fca,9d4ec75d0c453a4d
Lung_Opacity-1825.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1825.png,Lung_Opacity/masks/Lung_Opacity-1825.png,True,G04320,train,3c730892c131e83982d509c8966336bc,916a43161e595d7e
Lung_Opacity-1826.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1826.png,Lung_Opacity/masks/Lung_Opacity-1826.png,True,G04321,train,c175711f176d41ec5f9784cb1d5a6fcf,9c3f82035f075f0b
Lung_Opacity-1827.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1827.png,Lung_Opacity/masks/Lung_Opacity-1827.png,True,G04322,train,e5b57e2bfa387b1b4fd9a2e160ba6726,8b380f0f5a6978f8
Lung_Opacity-1828.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1828.png,Lung_Opacity/masks/Lung_Opacity-1828.png,True,G04323,train,47eec52659cb21e0ab87870908fa2232,d578164a5a474f4b
Lung_Opacity-1829.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1829.png,Lung_Opacity/masks/Lung_Opacity-1829.png,True,G04324,train,c9ad80e5bc3d008c85ef4cbaef32b6e3,9c2a970b4f4f0d4d
Lung_Opacity-183.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-183.png,Lung_Opacity/masks/Lung_Opacity-183.png,True,G04325,train,329f4da38dacf74023220e752497cfd6,8939c21e1e5b5a5b
Lung_Opacity-1830.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1830.png,Lung_Opacity/masks/Lung_Opacity-1830.png,True,G04326,train,a277b5aae1159ff9f5617191974d9146,98212d1d1f1d1d5f
Lung_Opacity-1831.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1831.png,Lung_Opacity/masks/Lung_Opacity-1831.png,True,G04327,train,879f689a2590474fe646b806482038a4,d86cc21b455b595b
Lung_Opacity-1832.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1832.png,Lung_Opacity/masks/Lung_Opacity-1832.png,True,G04328,train,19656e50eea0ccf92ddfd7025a96d871,817ae84f9037525f
Lung_Opacity-1833.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1833.png,Lung_Opacity/masks/Lung_Opacity-1833.png,True,G04329,train,7b472bbba175cec056e4da12b4daae38,c42aa50dae2e2f75
Lung_Opacity-1834.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1834.png,Lung_Opacity/masks/Lung_Opacity-1834.png,True,G04330,train,2b045646dbc7c16824cee0edb09ffe2b,c03b3f3c1e6c6c64
Lung_Opacity-1835.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1835.png,Lung_Opacity/masks/Lung_Opacity-1835.png,True,G04331,train,955466885b885c041da03185a8314bb5,d82985161f4f1e5d
Lung_Opacity-1836.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1836.png,Lung_Opacity/masks/Lung_Opacity-1836.png,True,G04332,train,63a5658d1ee8e865bd37e5ac60a5591c,8169571e585f3939
Lung_Opacity-1837.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1837.png,Lung_Opacity/masks/Lung_Opacity-1837.png,True,G04333,train,ed67c719a6d7192cbbd1e3e77e2aa9ae,8179044c1e1f3f7d
Lung_Opacity-1838.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1838.png,Lung_Opacity/masks/Lung_Opacity-1838.png,True,G04334,val,8b02751d3a036d2cc46b018a54e0838a,dc2d534b0f0e3c1e
Lung_Opacity-1839.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1839.png,Lung_Opacity/masks/Lung_Opacity-1839.png,True,G04335,val,4ec15d57c5786eab516141d04bc1f638,817f06165a5f5a59
Lung_Opacity-184.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-184.png,Lung_Opacity/masks/Lung_Opacity-184.png,True,G04336,val,6749b9d2eb5302ea4c6457d9b374a86e,817b96ca19195a7b
Lung_Opacity-1840.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1840.png,Lung_Opacity/masks/Lung_Opacity-1840.png,True,G04337,test,675ebed9c859ffa3e75117b7c4d1179c,dd2ad34b4d0a4f4c
Lung_Opacity-1841.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1841.png,Lung_Opacity/masks/Lung_Opacity-1841.png,True,G04338,test,0434cf7d5fc8b358097a82aaaf866b12,997b061e5a5b5958
Lung_Opacity-1842.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1842.png,Lung_Opacity/masks/Lung_Opacity-1842.png,True,G04339,test,bd1fbf4310ae5d5ed37001352d23c964,917987161a0f3f1b
Lung_Opacity-1843.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1843.png,Lung_Opacity/masks/Lung_Opacity-1843.png,True,G04340,train,c0811fb7d02b7b29d5095a1c5aac7407,d57f42025a4f1e4e
Lung_Opacity-1844.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1844.png,Lung_Opacity/masks/Lung_Opacity-1844.png,True,G04341,train,e46c68e09ee2f4d540fbc64b32a02069,852916065f5efa78
Lung_Opacity-1845.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1845.png,Lung_Opacity/masks/Lung_Opacity-1845.png,True,G04342,train,ca58e5d231cebd2ada0b4a4ff1e600e3,d42d494f4e2d0f2d
Lung_Opacity-1846.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1846.png,Lung_Opacity/masks/Lung_Opacity-1846.png,True,G04343,train,e01c18168a0951469589af95a4f9b339,9d4e167a6172b2b1
Lung_Opacity-1847.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1847.png,Lung_Opacity/masks/Lung_Opacity-1847.png,True,G04344,train,ff7eb7a228f02e140e508120bcfc770f,c0344f4b3d4f0e3d
Lung_Opacity-1848.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1848.png,Lung_Opacity/masks/Lung_Opacity-1848.png,True,G04345,val,11231d973801d7cf19ba430f75b7c9cc,d42d5b0cae0e0f97
Lung_Opacity-1849.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1849.png,Lung_Opacity/masks/Lung_Opacity-1849.png,True,G04346,train,9d1ffdb09dec22fb3fbd041ba96a046a,957d475f59182929
Lung_Opacity-185.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-185.png,Lung_Opacity/masks/Lung_Opacity-185.png,True,G04347,train,e9258baf53e054108afb58a4569d62f1,c53e121a1f697e62
Lung_Opacity-1850.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1850.png,Lung_Opacity/masks/Lung_Opacity-1850.png,True,G04348,train,cb01ee9917b3403562195e9d012e9990,802e475f1f5c4e4e
Lung_Opacity-1851.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1851.png,Lung_Opacity/masks/Lung_Opacity-1851.png,True,G04349,train,7495626562af0d831fc7130c21f67f5f,dd28c74b282c2fb5
Lung_Opacity-1852.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1852.png,Lung_Opacity/masks/Lung_Opacity-1852.png,True,G04350,train,995c0d013cc957b6a44fd3d1ba71124e,d57ffe1450555501
Lung_Opacity-1853.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1853.png,Lung_Opacity/masks/Lung_Opacity-1853.png,True,G04351,train,22d4c8991047a4b3ed03042e109e4381,cd2e5343495c4d4f
Lung_Opacity-1854.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1854.png,Lung_Opacity/masks/Lung_Opacity-1854.png,True,G04352,train,1470a80bb46376b436c88ff4076857ea,d13dc7d21b58495c
Lung_Opacity-1855.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1855.png,Lung_Opacity/masks/Lung_Opacity-1855.png,True,G04353,train,411fa0b7fba93c02482c4235032b3570,d43f431d1c4a1b1f
Lung_Opacity-1856.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1856.png,Lung_Opacity/masks/Lung_Opacity-1856.png,True,G04354,train,f28bca8778fef748d30cbde683ed2591,997f865a185e5a1a
Lung_Opacity-1857.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1857.png,Lung_Opacity/masks/Lung_Opacity-1857.png,True,G04355,val,224d44644bd364b920e1defb4392e617,91601e171e5f5c7c
Lung_Opacity-1858.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1858.png,Lung_Opacity/masks/Lung_Opacity-1858.png,True,G04356,test,2b64d359b1dc8465ce02d58bff164925,c03d4a0b1f3d3d3c
Lung_Opacity-1859.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1859.png,Lung_Opacity/masks/Lung_Opacity-1859.png,True,G04357,test,a7254ce76e480319eea3123272ce346c,9128479e1d5f1b1b
Lung_Opacity-186.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-186.png,Lung_Opacity/masks/Lung_Opacity-186.png,True,G04358,train,10450ef8cf5f14c4134729d302fac6bc,d42f437f30733330
Lung_Opacity-1860.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1860.png,Lung_Opacity/masks/Lung_Opacity-1860.png,True,G04359,val,6abc432f2be93662dec8a0b0e0d1e66e,89788e5e587968e9
Lung_Opacity-1861.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1861.png,Lung_Opacity/masks/Lung_Opacity-1861.png,True,G04360,test,e6f2e1224dce0300b003b3063c880bbb,917b101b1b793db2
Lung_Opacity-1862.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1862.png,Lung_Opacity/masks/Lung_Opacity-1862.png,True,G04361,train,d5fea4e63a7c8168c6c6c98cd5dfd7ee,93614c5d5e5a396a
Lung_Opacity-1863.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1863.png,Lung_Opacity/masks/Lung_Opacity-1863.png,True,G04362,train,3befb54112cdc63707d242d6b61b5dec,8c6c83c78b4f095f
Lung_Opacity-1864.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1864.png,Lung_Opacity/masks/Lung_Opacity-1864.png,True,G04363,val,0c4ab85a2d95d670f92e45d30c5df6cb,933940185f7f165e
Lung_Opacity-1865.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1865.png,Lung_Opacity/masks/Lung_Opacity-1865.png,True,G04364,train,bccf288286c6eb1f4fb2672bb1b62a60,dd2a561e4d287969
Lung_Opacity-1866.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1866.png,Lung_Opacity/masks/Lung_Opacity-1866.png,True,G04365,train,82ada0385271619e9987ce2a7b666499,917b86061e1b2f3d
Lung_Opacity-1867.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1867.png,Lung_Opacity/masks/Lung_Opacity-1867.png,True,G04366,train,8df75f9601885c7b1cac9013b43778e3,dc2dc20b0e6c3fb8
Lung_Opacity-1868.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1868.png,Lung_Opacity/masks/Lung_Opacity-1868.png,True,G04367,train,ddc1d9f9e16f873cbdb93847086ad85f,9978961b19197b53
Lung_Opacity-1869.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1869.png,Lung_Opacity/masks/Lung_Opacity-1869.png,True,G04368,train,3b9d9cda6342f786fe2e6b3947923cc3,d02bd75209791f1b
Lung_Opacity-187.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-187.png,Lung_Opacity/masks/Lung_Opacity-187.png,True,G04369,train,34d7475a8d4f4d545c015558806e0bde,8179d61a5a4a397d
Lung_Opacity-1870.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1870.png,Lung_Opacity/masks/Lung_Opacity-1870.png,True,G04370,train,4a23959e9a3a638b351c96fae5cff594,917ec7521b4b1e19
Lung_Opacity-1871.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1871.png,Lung_Opacity/masks/Lung_Opacity-1871.png,True,G04371,train,a60e549cd037373adf829d86f0c2df7b,d42f034b0f0f2b2f
Lung_Opacity-1872.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1872.png,Lung_Opacity/masks/Lung_Opacity-1872.png,True,G04372,train,f38d79dd1efb98c3b4c71f91e4700ae6,dc3d434b4b0b0e2f
Lung_Opacity-1873.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1873.png,Lung_Opacity/masks/Lung_Opacity-1873.png,True,G04373,train,4111a9755ad05edc75217a0b8370481c,8368151f587b7939
Lung_Opacity-1874.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1874.png,Lung_Opacity/masks/Lung_Opacity-1874.png,True,G04374,train,43888006f9ee226425fa948bc8d53add,9d681e737364468e
Lung_Opacity-1875.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1875.png,Lung_Opacity/masks/Lung_Opacity-1875.png,True,G04375,val,d18edf7b1b2bdf5642a8526cd2da1e66,cc39d3435e4b0b69
Lung_Opacity-1876.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1876.png,Lung_Opacity/masks/Lung_Opacity-1876.png,True,G04376,train,f62f6b1a94a2ecee5aa886129fa57431,c43c4a4aecf0f3e1
Lung_Opacity-1877.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1877.png,Lung_Opacity/masks/Lung_Opacity-1877.png,True,G04377,val,5282e61cbb0b29e6b8075ebf00cf89db,c42e43431b2f6e6e
Lung_Opacity-1878.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1878.png,Lung_Opacity/masks/Lung_Opacity-1878.png,True,G04378,test,286f33cfc1844ab85a7c13adeaafd0f7,dc2883c39b578753
Lung_Opacity-1879.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1879.png,Lung_Opacity/masks/Lung_Opacity-1879.png,True,G04379,test,0f164c414e620ba7355e20c1f509474d,956c0f5e4e6ae864
Lung_Opacity-188.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-188.png,Lung_Opacity/masks/Lung_Opacity-188.png,True,G04380,train,370f8b6b8392af29daffb7def49dc81c,8178481f1b79787e
Lung_Opacity-1880.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1880.png,Lung_Opacity/masks/Lung_Opacity-1880.png,True,G04381,test,53206597d5a73768ddc105beebafc4cf,81e8ee869859c7f8
Lung_Opacity-1881.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1881.png,Lung_Opacity/masks/Lung_Opacity-1881.png,True,G04382,train,add923f394df1b7730154d4dfdfe9421,d02b535a19591f1f
Lung_Opacity-1882.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1882.png,Lung_Opacity/masks/Lung_Opacity-1882.png,True,G04383,train,81951afb85087353fab376172506af0e,c12b561b583d3d0f
Lung_Opacity-1883.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1883.png,Lung_Opacity/masks/Lung_Opacity-1883.png,True,G04384,train,57e2db5ac36860b0a2a089468b5af46b,9d6c07121b5d7e58
Lung_Opacity-1884.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1884.png,Lung_Opacity/masks/Lung_Opacity-1884.png,True,G04385,train,c35525300a4230ddfc6fdfacd49408e5,9b7816101b35a7bd
Lung_Opacity-1885.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1885.png,Lung_Opacity/masks/Lung_Opacity-1885.png,True,G04386,train,e190d086277cb9fb6c95928322ffa75c,c179875a183f4d39
Lung_Opacity-1886.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1886.png,Lung_Opacity/masks/Lung_Opacity-1886.png,True,G04387,train,300652fb3a934372f6f33c477f977206,8168c716c75f1e1e
Lung_Opacity-1887.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1887.png,Lung_Opacity/masks/Lung_Opacity-1887.png,True,G04388,train,d2f1573c2b8376141c0c786b13b5441c,d06b561e4d1d1f0b
Lung_Opacity-1888.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1888.png,Lung_Opacity/masks/Lung_Opacity-1888.png,True,G04389,train,10307ac84e0d35054b9374d6e80762a3,c52ec34b0d6c2d6d
Lung_Opacity-1889.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1889.png,Lung_Opacity/masks/Lung_Opacity-1889.png,True,G04390,train,4566d8f866b3ff929db4573fa577affe,907fc45e18292fae
Lung_Opacity-189.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-189.png,Lung_Opacity/masks/Lung_Opacity-189.png,True,G04391,train,a3e6df76756d463854fcb425d5836c40,c6264a4f66e6e4f0
Lung_Opacity-1890.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1890.png,Lung_Opacity/masks/Lung_Opacity-1890.png,True,G04392,train,dea1372b92ddf78e46d461486136b47f,d83dc7440d479e1b
Lung_Opacity-1891.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1891.png,Lung_Opacity/masks/Lung_Opacity-1891.png,True,G04393,train,e62018092797e659618a784dbdeec1ec,883f870e0f0f1f0f
Lung_Opacity-1892.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1892.png,Lung_Opacity/masks/Lung_Opacity-1892.png,True,G04394,train,2675e36e6e300eb2b3e9c25de08b4e8c,99296bd86696662d
Lung_Opacity-1893.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1893.png,Lung_Opacity/masks/Lung_Opacity-1893.png,True,G04395,train,cb746e7e75de1ec582f8d14691711080,d50b1a1d0deb25cb
Lung_Opacity-1894.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1894.png,Lung_Opacity/masks/Lung_Opacity-1894.png,True,G04396,val,11d0e6b36f1f8768e8598433cbead966,d02d2fd42f8a2fc2
Lung_Opacity-1895.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1895.png,Lung_Opacity/masks/Lung_Opacity-1895.png,True,G04397,val,3e0077075ebbf82e9ca940e75f0cd01d,9968161a4f1a5b5f
Lung_Opacity-1896.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1896.png,Lung_Opacity/masks/Lung_Opacity-1896.png,True,G04398,val,93f6ec5ebd8e275f70ef4bced5712647,c13f564b0d0b2d3d
Lung_Opacity-1897.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1897.png,Lung_Opacity/masks/Lung_Opacity-1897.png,True,G04399,test,a8f529cb83cfd24caf74d2e3cdaf6d57,856c06c24b3dedcb
Lung_Opacity-1898.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1898.png,Lung_Opacity/masks/Lung_Opacity-1898.png,True,G04400,test,fa7932ee43cadc0ef2e54d6533a4e6e9,817bd786944a7a5a
Lung_Opacity-1899.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1899.png,Lung_Opacity/masks/Lung_Opacity-1899.png,True,G04401,test,eda7cdba4e3e67e2d37f35d695b8f806,9139c25f595b1b19
Lung_Opacity-19.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-19.png,Lung_Opacity/masks/Lung_Opacity-19.png,True,G04402,train,cfac9df791266a4622b798f8b30d7364,9968151e5af9c9ca
Lung_Opacity-190.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-190.png,Lung_Opacity/masks/Lung_Opacity-190.png,True,G04403,train,b69d2f4b2e52d8fcdc218e5134440ff0,c52f160b4d4a4d6f
Lung_Opacity-1900.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1900.png,Lung_Opacity/masks/Lung_Opacity-1900.png,True,G04404,train,6f379256374f786424f18adedd9591df,91780f1e39796969
Lung_Opacity-1901.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1901.png,Lung_Opacity/masks/Lung_Opacity-1901.png,True,G04405,train,6c79e2b7adf005f7ac7b4c7a06b31ed6,d627c1494b0f0f2f
Lung_Opacity-1902.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1902.png,Lung_Opacity/masks/Lung_Opacity-1902.png,True,G04406,train,04d80526ce7a13f6fc028c68cb5efe5a,c53c090d3f13b7b2
Lung_Opacity-1903.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1903.png,Lung_Opacity/masks/Lung_Opacity-1903.png,True,G04407,train,09efa95c8d93b26a4d8abaa6713e60ea,d43d22d20bd34f4e
Lung_Opacity-1904.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1904.png,Lung_Opacity/masks/Lung_Opacity-1904.png,True,G04408,train,bd6b51fbfc499382ad5ff7734f978e09,d951d61e5c535958
Lung_Opacity-1905.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1905.png,Lung_Opacity/masks/Lung_Opacity-1905.png,True,G04409,train,1945e2256908a81e86852a6c06d1646c,cd2c871b1c4e1d6d
Lung_Opacity-1906.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1906.png,Lung_Opacity/masks/Lung_Opacity-1906.png,True,G04410,train,f94ae94ad30bc90e44a4fc78a3e0029a,913d45421e5f4b2f
Lung_Opacity-1907.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1907.png,Lung_Opacity/masks/Lung_Opacity-1907.png,True,G04411,train,e71afd01239ec5094a9e642a49ebc171,9979d75b591a4819
Lung_Opacity-1908.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1908.png,Lung_Opacity/masks/Lung_Opacity-1908.png,True,G04412,test,129a5303abe2f0d23feb113f462325f7,89384f571c4fa479
Lung_Opacity-1909.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1909.png,Lung_Opacity/masks/Lung_Opacity-1909.png,True,G04413,train,f564e90e41e48f3f6e5284f4a3e291eb,99684a1f181f7d3c
Lung_Opacity-191.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-191.png,Lung_Opacity/masks/Lung_Opacity-191.png,True,G04414,train,821815e72ea30c1cf15abf03e5866d5a,917a431e1e6939f4
Lung_Opacity-1910.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1910.png,Lung_Opacity/masks/Lung_Opacity-1910.png,True,G04415,train,e71e69b0386a7a5b0097cd152be6c513,95694a9439697cf2
Lung_Opacity-1911.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1911.png,Lung_Opacity/masks/Lung_Opacity-1911.png,True,G04416,val,f47c82561db2ff5a91aa245cf23cfa94,9178065e1a7b7a63
Lung_Opacity-1912.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1912.png,Lung_Opacity/masks/Lung_Opacity-1912.png,True,G04417,val,f9d13885abbd11028fd3d46b7ff79adb,817a6a56125b3e5b
Lung_Opacity-1913.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1913.png,Lung_Opacity/masks/Lung_Opacity-1913.png,True,G04418,test,7d6cb7cab33716db3ce6837e0bbdf032,d5204dce1a5e4f4e
Lung_Opacity-1914.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1914.png,Lung_Opacity/masks/Lung_Opacity-1914.png,True,G04419,train,d0851510ffe10dc7d027a685aee1fd10,cc2e49092eadbab3
Lung_Opacity-1915.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1915.png,Lung_Opacity/masks/Lung_Opacity-1915.png,True,G04420,val,6f46a31a439ad16a17e30d5375acd324,c0d43f0b7d35c274
Lung_Opacity-1916.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1916.png,Lung_Opacity/masks/Lung_Opacity-1916.png,True,G04421,train,600fe80f56311edeb0fa99712e2eeb4a,d42d4b4f2e2e0d1e
Lung_Opacity-1917.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1917.png,Lung_Opacity/masks/Lung_Opacity-1917.png,True,G04422,train,a919caa13be8696aab46907018334b24,817d06561f5a1f5a
Lung_Opacity-1918.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1918.png,Lung_Opacity/masks/Lung_Opacity-1918.png,True,G04423,train,f3e4e923d5aaf7c665248ce15bdea94d,d42e87170d4d4d4f
Lung_Opacity-1919.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1919.png,Lung_Opacity/masks/Lung_Opacity-1919.png,True,G04424,train,7f6096bb158509bc15156c8f1dfd4425,817a505b5b5b5959
Lung_Opacity-192.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-192.png,Lung_Opacity/masks/Lung_Opacity-192.png,True,G04425,train,9b0a8d64f87ee898a60d0b9fa2120ae8,d52f414b2f727838
Lung_Opacity-1920.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1920.png,Lung_Opacity/masks/Lung_Opacity-1920.png,True,G04426,test,e73ae88b1eba016772656799014f788f,c9366a48282f2f3f
Lung_Opacity-1921.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1921.png,Lung_Opacity/masks/Lung_Opacity-1921.png,True,G04427,train,2f0566e84ae6adace7de88007de06932,943c6b43574e550f
Lung_Opacity-1922.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1922.png,Lung_Opacity/masks/Lung_Opacity-1922.png,True,G04428,train,b81f9b206ec38cbabcf8b37363c4461b,986f861a0d3c4f1f
Lung_Opacity-1923.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1923.png,Lung_Opacity/masks/Lung_Opacity-1923.png,True,G04429,train,ff2fe24eae72cd0acde3dc0398deaad8,952e07871f5f0d45
Lung_Opacity-1924.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1924.png,Lung_Opacity/masks/Lung_Opacity-1924.png,True,G04430,train,90bcbd668f133cc6284877dac333817d,8b78061e185b5b5f
Lung_Opacity-1925.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1925.png,Lung_Opacity/masks/Lung_Opacity-1925.png,True,G04431,train,07fc2739d2b2da4af0dd3b064cbe0cb0,857e2d07074f0f4e
Lung_Opacity-1926.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1926.png,Lung_Opacity/masks/Lung_Opacity-1926.png,True,G04432,train,a5a571083b0a453fe02790a3458afa70,d42d831f086dbdb8
Lung_Opacity-1927.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1927.png,Lung_Opacity/masks/Lung_Opacity-1927.png,True,G04433,train,de9e7cde0db9afc8915f4cf48afb0f32,c13996d21f665a1d
Lung_Opacity-1928.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1928.png,Lung_Opacity/masks/Lung_Opacity-1928.png,True,G04434,train,816d9199dfe87364d45e3ddbcaf85467,d52d051a2be8f1f2
Lung_Opacity-1929.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1929.png,Lung_Opacity/masks/Lung_Opacity-1929.png,True,G04435,val,d67877d207aec91654c5fce75f4e7022,8178441e795b7b1b
Lung_Opacity-193.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-193.png,Lung_Opacity/masks/Lung_Opacity-193.png,True,G04436,val,79d383b93c7660821646b721d8533da2,9939ce16190ded39
Lung_Opacity-1930.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1930.png,Lung_Opacity/masks/Lung_Opacity-1930.png,True,G04437,test,781e74170a766a396a47a1b79c7e16c0,897d841e184f5d5e
Lung_Opacity-1931.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1931.png,Lung_Opacity/masks/Lung_Opacity-1931.png,True,G04438,val,9dbe6771c273647f9ab601c08cae95e3,817905865f927ecd
Lung_Opacity-1932.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1932.png,Lung_Opacity/masks/Lung_Opacity-1932.png,True,G04439,test,31550a4bd98d57e8047e4fac7eb49771,d03b3ac43f903dd2
Lung_Opacity-1933.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1933.png,Lung_Opacity/masks/Lung_Opacity-1933.png,True,G04440,test,0edb2c79ba79980ee7b554ce8be9696f,d02e0c4c4f69e1f7
Lung_Opacity-1934.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1934.png,Lung_Opacity/masks/Lung_Opacity-1934.png,True,G04441,train,f9b38596309a54f39629e786e5248177,813a4e1e7bede805
Lung_Opacity-1935.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1935.png,Lung_Opacity/masks/Lung_Opacity-1935.png,True,G04442,train,b89d787f8dbacc79bbdc47f3b45269b2,c83d4b0e29293dbb
Lung_Opacity-1936.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1936.png,Lung_Opacity/masks/Lung_Opacity-1936.png,True,G04443,val,26c0c1393b931f55b919f549e6c3edc9,cd2c16c24b1f4e5e
Lung_Opacity-1937.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1937.png,Lung_Opacity/masks/Lung_Opacity-1937.png,True,G04444,train,32bb560755b65ee2d3d3f576c0883dd6,c23e0f0e0d2f4d1f
Lung_Opacity-1938.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1938.png,Lung_Opacity/masks/Lung_Opacity-1938.png,True,G04445,train,9406bac43adfe920113145a76cad0883,cd2d40131b4d2e7f
Lung_Opacity-1939.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1939.png,Lung_Opacity/masks/Lung_Opacity-1939.png,True,G04446,train,5f95396403a13cbe71de08105e66403f,d069080f1c5f5f5d
Lung_Opacity-194.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-194.png,Lung_Opacity/masks/Lung_Opacity-194.png,True,G04447,train,caf6ac4c5588af5770374e42ab609444,894e1659491f3e3e
Lung_Opacity-1940.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1940.png,Lung_Opacity/masks/Lung_Opacity-1940.png,True,G04448,train,f9c5312eec9a298afc067cb64a2ea6cd,803b1a1b3bbab4b6
Lung_Opacity-1941.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1941.png,Lung_Opacity/masks/Lung_Opacity-1941.png,True,G04449,train,8d25f104fd27b09e85ea514b3f332dca,d43d43430b3c3d3d
Lung_Opacity-1942.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1942.png,Lung_Opacity/masks/Lung_Opacity-1942.png,True,G04450,train,5a28eee676865f6e0b713b6659bfb6cb,dc68c30f5d1c5c1b
Lung_Opacity-1943.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1943.png,Lung_Opacity/masks/Lung_Opacity-1943.png,True,G04451,train,71566762b0d196178d3c5e6d91a4c7e0,817f3f0a881e9a1f
Lung_Opacity-1944.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1944.png,Lung_Opacity/masks/Lung_Opacity-1944.png,True,G04452,train,f5947ad7118b4579a303b594c258f174,992d161e495d515f
Lung_Opacity-1945.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1945.png,Lung_Opacity/masks/Lung_Opacity-1945.png,True,G04453,train,bcb13d5b36233df0b2294b1d472d8e8a,dc28430f5c1acfba
Lung_Opacity-1946.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1946.png,Lung_Opacity/masks/Lung_Opacity-1946.png,True,G04454,train,00030c4aedde54573684165d26089bc1,990f861e58697ce5
Lung_Opacity-1947.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1947.png,Lung_Opacity/masks/Lung_Opacity-1947.png,True,G04455,train,fb7adeecd1f4e2d0187c46ae008840d0,c03d52434f4f2e2f
Lung_Opacity-1948.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1948.png,Lung_Opacity/masks/Lung_Opacity-1948.png,True,G04456,train,2379b543d256ff26240f8baf7989a600,c13b16165e1a7b2b
Lung_Opacity-1949.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1949.png,Lung_Opacity/masks/Lung_Opacity-1949.png,True,G04457,val,ef6fd26fad934c3f9173ab269977f148,dd7da4c0929bc3c1
Lung_Opacity-195.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-195.png,Lung_Opacity/masks/Lung_Opacity-195.png,True,G04458,val,4bdfeaaf6f6dbcd33b0b95b3bd4d4dd3,99584a19bd19bf0d
Lung_Opacity-1950.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1950.png,Lung_Opacity/masks/Lung_Opacity-1950.png,True,G04459,test,386dab21b900b5da515fa3eb3f597e0e,dd2fc20b0cae0de5
Lung_Opacity-1951.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1951.png,Lung_Opacity/masks/Lung_Opacity-1951.png,True,G04460,test,1ede51d2c54ab27ca5bf4f89e20a3861,9d281f1e5a78e0e6
Lung_Opacity-1952.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1952.png,Lung_Opacity/masks/Lung_Opacity-1952.png,True,G04461,test,d3261984d78045a84e49c8c13ed40c43,837f141a7a5b1b1a
Lung_Opacity-1953.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1953.png,Lung_Opacity/masks/Lung_Opacity-1953.png,True,G04462,train,a938393511f00d01a698076358b1b1b7,99695e1a6bb8e125
Lung_Opacity-1954.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1954.png,Lung_Opacity/masks/Lung_Opacity-1954.png,True,G04463,train,84688897a51e248844015683409858c3,807cef0b0b4f0f4d
Lung_Opacity-1955.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1955.png,Lung_Opacity/masks/Lung_Opacity-1955.png,True,G04464,train,f626c6e6464bb9872831bef247897d41,d62f054e0a6e2e6e
Lung_Opacity-1956.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1956.png,Lung_Opacity/masks/Lung_Opacity-1956.png,True,G04465,train,5928c76b78c68a9bd730c84215a8b0fc,9d6c475f4d6868e0
Lung_Opacity-1957.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1957.png,Lung_Opacity/masks/Lung_Opacity-1957.png,True,G04466,train,1d841e18672fffd578b78a5de8ef3cab,c94d861a194d5f1f
Lung_Opacity-1958.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1958.png,Lung_Opacity/masks/Lung_Opacity-1958.png,True,G04467,train,029dc5a4aebfd61415d5d43a3919cab2,dd2a56135b0d2d1d
Lung_Opacity-1959.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1959.png,Lung_Opacity/masks/Lung_Opacity-1959.png,True,G04468,train,bdf04f9e4f1f76249ea118d928119abe,c13b541259597d5b
Lung_Opacity-196.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-196.png,Lung_Opacity/masks/Lung_Opacity-196.png,True,G04469,train,fca8de2a3fd99d574c9d11948a86167e,903fe6476f3a3848
Lung_Opacity-1960.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1960.png,Lung_Opacity/masks/Lung_Opacity-1960.png,True,G04470,test,2667d4dc83227690976a4736e1db99f8,cd281e4c5cece4e9
Lung_Opacity-1961.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1961.png,Lung_Opacity/masks/Lung_Opacity-1961.png,True,G04471,train,d1b3690b2cc01de6c8f73259ef7269cf,d47c35cb38c43ccc
Lung_Opacity-1962.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1962.png,Lung_Opacity/masks/Lung_Opacity-1962.png,True,G04472,val,6bfcdfb763f70d906d08f689f8d42f23,8378141a5b5b3b7a
Lung_Opacity-1963.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1963.png,Lung_Opacity/masks/Lung_Opacity-1963.png,True,G04473,train,1c1f03f8f2b716e41589ae236bf52c73,d529420f0a3a3f3f
Lung_Opacity-1964.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1964.png,Lung_Opacity/masks/Lung_Opacity-1964.png,True,G04474,test,ae3dbc7eb7bc379d2f7e36a7950668a7,d42d3bea1ecc38c8
Lung_Opacity-1965.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1965.png,Lung_Opacity/masks/Lung_Opacity-1965.png,True,G04475,train,be95b66b9c9be64f12943d8ae50aac61,9762541e7a7a2a2b
Lung_Opacity-1966.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1966.png,Lung_Opacity/masks/Lung_Opacity-1966.png,True,G04476,train,44b45c95b74f8b1ef9e47133ba178aa7,8178081b5f597b79
Lung_Opacity-1967.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1967.png,Lung_Opacity/masks/Lung_Opacity-1967.png,True,G04477,train,b1034ad4d115bd96a669fd1aecf8c1a6,d87bc5d616417e48
Lung_Opacity-1968.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1968.png,Lung_Opacity/masks/Lung_Opacity-1968.png,True,G04478,val,c8fe2b3b158edf34b3d0cd753556c454,8d2e8349455d4b5f
Lung_Opacity-1969.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1969.png,Lung_Opacity/masks/Lung_Opacity-1969.png,True,G04479,train,f835782d82f7fddb123368e09a7b37f5,d62fd5510e2a0f0f
Lung_Opacity-197.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-197.png,Lung_Opacity/masks/Lung_Opacity-197.png,True,G04480,val,15b2a21b1192d08be5d80ae383f2e98c,946eaf3d341c2d0d
Lung_Opacity-1970.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1970.png,Lung_Opacity/masks/Lung_Opacity-1970.png,True,G04481,test,799634ebe3a85bb49634eaee211b5b4c,e949061a195d5b5f
Lung_Opacity-1971.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1971.png,Lung_Opacity/masks/Lung_Opacity-1971.png,True,G04482,val,5c6ab3e7cfb8679692fb542afeca200d,9578c7561a49cb1e
Lung_Opacity-1972.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1972.png,Lung_Opacity/masks/Lung_Opacity-1972.png,True,G04483,train,0beddaa09c7ed8cec3f7c3c5fc6096ba,ca35d11b4b6a6c6c
Lung_Opacity-1973.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1973.png,Lung_Opacity/masks/Lung_Opacity-1973.png,True,G04484,train,8c5e0c4becbc558537539803e1e6190f,d03bc307474b1a7b
Lung_Opacity-1974.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1974.png,Lung_Opacity/masks/Lung_Opacity-1974.png,True,G04485,train,261f61db5ab4a2d55212e9d6ede50985,d46f4b0c1cabb1b4
Lung_Opacity-1975.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1975.png,Lung_Opacity/masks/Lung_Opacity-1975.png,True,G04486,train,7a57de128b40ced1db9b05b659c931b9,d1381e1f7879e2c2
Lung_Opacity-1976.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1976.png,Lung_Opacity/masks/Lung_Opacity-1976.png,True,G04487,train,4cb2a85b592459f17abc92884471e19c,c13d1a3f786c694a
Lung_Opacity-1977.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1977.png,Lung_Opacity/masks/Lung_Opacity-1977.png,True,G04488,train,b709998280b29edf8f8565f5fcb42f6b,d13f46570c1d1d59
Lung_Opacity-1978.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1978.png,Lung_Opacity/masks/Lung_Opacity-1978.png,True,G04489,train,c4c92b94e0d9f563ed49ba13ae9afffd,957c040c1f5d4f5e
Lung_Opacity-1979.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1979.png,Lung_Opacity/masks/Lung_Opacity-1979.png,True,G04490,train,aca5dd1b5e885ef6bc669aedf04e0275,9f68041f943e383f
Lung_Opacity-198.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-198.png,Lung_Opacity/masks/Lung_Opacity-198.png,True,G04491,train,4cbd317442fc1444daf985af0aa7541c,9568a73333c69cc9
Lung_Opacity-1980.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1980.png,Lung_Opacity/masks/Lung_Opacity-1980.png,True,G04492,train,15285d6c6522b47ae3bc50d2a00b1804,d42e490d2e3e98db
Lung_Opacity-1981.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1981.png,Lung_Opacity/masks/Lung_Opacity-1981.png,True,G04493,train,9c9b7c7a89a796eb526c91899f8f4aff,c3315e5c584f1d9c
Lung_Opacity-1982.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1982.png,Lung_Opacity/masks/Lung_Opacity-1982.png,True,G04494,train,c2591b7e8aa6be49cd256d4060e1069e,9528541f5a59795b
Lung_Opacity-1983.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1983.png,Lung_Opacity/masks/Lung_Opacity-1983.png,True,G04495,train,e26d3195407ff9ff1f8f6ab7202e7d28,817f161a5e4d5879
Lung_Opacity-1984.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1984.png,Lung_Opacity/masks/Lung_Opacity-1984.png,True,G04496,train,f7d05830fac3e4224e5fe17211f7d931,817f845e595b4a78
Lung_Opacity-1985.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1985.png,Lung_Opacity/masks/Lung_Opacity-1985.png,True,G04497,train,82279d79ebd9652581826f322b3e1570,c93d56c31a1c8ecb
Lung_Opacity-1986.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1986.png,Lung_Opacity/masks/Lung_Opacity-1986.png,True,G04498,val,efe1e4b9779e64daeb3bec97b7b3c68a,d03d5f1b4a0b4b0f
Lung_Opacity-1987.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1987.png,Lung_Opacity/masks/Lung_Opacity-1987.png,True,G04499,val,15dc43d1b5f2cd1e57ac66d88b9f7df5,d42f4f590c2c3e1e
Lung_Opacity-1988.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1988.png,Lung_Opacity/masks/Lung_Opacity-1988.png,True,G04500,val,52249161f8af4d536d35badf92ab225d,9d7c475a1b0c2f29
Lung_Opacity-1989.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1989.png,Lung_Opacity/masks/Lung_Opacity-1989.png,True,G04501,test,4820ba706c8a24c5faeae5e66786d04a,813e0e5e7869ece8
Lung_Opacity-199.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-199.png,Lung_Opacity/masks/Lung_Opacity-199.png,True,G04502,test,b64f7b6a1ac3582b281921557befc7c3,da3d05070f1f1e2e
Lung_Opacity-1990.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1990.png,Lung_Opacity/masks/Lung_Opacity-1990.png,True,G04503,train,ba876ba65c00d36b89cb4895682da93d,85782e47861e595f
Lung_Opacity-1991.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1991.png,Lung_Opacity/masks/Lung_Opacity-1991.png,True,G04504,test,9ea8d7f7dc2b9edd12d6ef3c49e77e12,81689712895f795f
Lung_Opacity-1992.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1992.png,Lung_Opacity/masks/Lung_Opacity-1992.png,True,G04505,val,5e7a0c27adbebcc0b2bde18c0a260886,dc2d434f0f0e0e1f
Lung_Opacity-1993.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1993.png,Lung_Opacity/masks/Lung_Opacity-1993.png,True,G04506,val,d8945efbde59c30e202cafaf3ba989a1,c86d85179a871f3a
Lung_Opacity-1994.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1994.png,Lung_Opacity/masks/Lung_Opacity-1994.png,True,G04507,train,98e9b09d3ae56778a2f65d5c582932b1,957a6e8578933a85
Lung_Opacity-1995.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1995.png,Lung_Opacity/masks/Lung_Opacity-1995.png,True,G04508,train,6cb8aebd5f368e50371fa7190ae73e70,c03d4b4f1f2d1c2d
Lung_Opacity-1996.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1996.png,Lung_Opacity/masks/Lung_Opacity-1996.png,True,G04509,train,f4d0ffe4281598e74c798374a061ee69,957c07521f0f1c5d
Lung_Opacity-1997.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1997.png,Lung_Opacity/masks/Lung_Opacity-1997.png,True,G04510,train,7a0b3a0f41dadb53f7bb496c5b1c1eaf,8c7c47431e5c4b4f
Lung_Opacity-1998.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1998.png,Lung_Opacity/masks/Lung_Opacity-1998.png,True,G04511,train,688b3d3672dc2d3877558fc0959feece,9f605e5a181b5b1b
Lung_Opacity-1999.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-1999.png,Lung_Opacity/masks/Lung_Opacity-1999.png,True,G04512,train,cd6461726fb3a00d4bd66d4153bc5b3d,81214797d61cdf55
Lung_Opacity-2.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2.png,Lung_Opacity/masks/Lung_Opacity-2.png,True,G04513,test,8dc48e422c049a919ad1f7f2c73da6b1,903b561a3b3b3467
Lung_Opacity-20.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-20.png,Lung_Opacity/masks/Lung_Opacity-20.png,True,G04514,test,266f501546698b14f90dff4abb7f9771,d02fc64b4f5d4ac4
Lung_Opacity-200.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-200.png,Lung_Opacity/masks/Lung_Opacity-200.png,True,G04515,train,90219141c06b769326fc40f7cdec5350,9968d25b1c1b4b3b
Lung_Opacity-2000.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2000.png,Lung_Opacity/masks/Lung_Opacity-2000.png,True,G04516,val,f984c8cd0b77131b4826594a63120dd4,cc2e43430f2a7b3b
Lung_Opacity-2001.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2001.png,Lung_Opacity/masks/Lung_Opacity-2001.png,True,G04517,train,1be36a6e059a91f88dc8456fa39d8d30,8c2dc3521a5d4f5d
Lung_Opacity-2002.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2002.png,Lung_Opacity/masks/Lung_Opacity-2002.png,True,G04518,test,f8b89c145dba11f801000f41caa3b509,d03d86829dbda5c3
Lung_Opacity-2003.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2003.png,Lung_Opacity/masks/Lung_Opacity-2003.png,True,G04519,train,17e310bf0d9a0184b6e514f6db2ae8c1,c03b6c525d515d5d
Lung_Opacity-2004.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2004.png,Lung_Opacity/masks/Lung_Opacity-2004.png,True,G04520,train,a0ee27ef11b4fc1426e904f10f66318c,cdb922967dfa6060
Lung_Opacity-2005.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2005.png,Lung_Opacity/masks/Lung_Opacity-2005.png,True,G04521,train,0c2bde7cd535ac4d298d3ce4b9607f49,956b3c6267a59299
Lung_Opacity-2006.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2006.png,Lung_Opacity/masks/Lung_Opacity-2006.png,True,G04522,train,d4e6f9ffba41f182b081384fc0c74595,9c3f431e1c49f89c
Lung_Opacity-2007.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2007.png,Lung_Opacity/masks/Lung_Opacity-2007.png,True,G04523,train,20b36ba17daff1d12ae08b7968baa781,802b575f0f3d386a
Lung_Opacity-2008.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2008.png,Lung_Opacity/masks/Lung_Opacity-2008.png,True,G04524,train,7cfd1c463e1e075c7e55b32a1eaebe05,9b68151e195f5a71
Lung_Opacity-2009.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2009.png,Lung_Opacity/masks/Lung_Opacity-2009.png,True,G04525,train,c4413e1c802698069f63118b1dcc4905,d02b4b3f0a293b3b
Lung_Opacity-201.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-201.png,Lung_Opacity/masks/Lung_Opacity-201.png,True,G04526,train,1cf527863a76cf704ddad84e0fe7ea1a,dd68d65b414d4a4d
Lung_Opacity-2010.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2010.png,Lung_Opacity/masks/Lung_Opacity-2010.png,True,G04527,train,f71d3d061349db04752cce63eca6214e,d42f431d1e4b1d4b
Lung_Opacity-2011.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2011.png,Lung_Opacity/masks/Lung_Opacity-2011.png,True,G04528,train,a92e311a1b9d49a2441e23da4a8a95d9,d42e87971d1c1e1e
Lung_Opacity-2012.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2012.png,Lung_Opacity/masks/Lung_Opacity-2012.png,True,G04529,train,0e04f9f237f3561fcd65fb0a89313eb7,9c2cc35b7c7862e2
Lung_Opacity-2013.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2013.png,Lung_Opacity/masks/Lung_Opacity-2013.png,True,G04530,train,84c93a12f520ccccc5152a91e83c3460,de3dc3404e4cce0f
Lung_Opacity-2014.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2014.png,Lung_Opacity/masks/Lung_Opacity-2014.png,True,G04531,train,13d86753b1f12e79fabe763814edb157,cc3d434b4c4f0e4f
Lung_Opacity-2015.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2015.png,Lung_Opacity/masks/Lung_Opacity-2015.png,True,G04532,train,9ec27ddd4cb91cf6b5750f53c45178f1,913b141e7a5b7b50
Lung_Opacity-2016.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2016.png,Lung_Opacity/masks/Lung_Opacity-2016.png,True,G04533,train,cdf0c1d5d8ef0b21d8044b79a6998bab,d57f020a1b5e4a5b
Lung_Opacity-2017.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2017.png,Lung_Opacity/masks/Lung_Opacity-2017.png,True,G04534,train,00fda914ba1ccd34ca1f0cf07dd7e4fa,8178061f5b5e5a5b
Lung_Opacity-2018.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2018.png,Lung_Opacity/masks/Lung_Opacity-2018.png,True,G04535,train,93e4ec8876b8309e16327d2b6ef3606e,d93bd65839672524
Lung_Opacity-2019.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2019.png,Lung_Opacity/masks/Lung_Opacity-2019.png,True,G04536,train,5ac5f2a3a4c435ee5f88f5baa7c688d1,c17f471a198e1b39
Lung_Opacity-202.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-202.png,Lung_Opacity/masks/Lung_Opacity-202.png,True,G04537,val,f4fd241b0f073d5a06436a3bf6547396,977094171f39391d
Lung_Opacity-2020.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2020.png,Lung_Opacity/masks/Lung_Opacity-2020.png,True,G04538,val,5c4030a503a41ff903d05def5357dbdb,842e434b4e6e6cee
Lung_Opacity-2021.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2021.png,Lung_Opacity/masks/Lung_Opacity-2021.png,True,G04539,val,845c624a2f9d980f6af09f8181d9c9db,81285f1e5e596979
Lung_Opacity-2022.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2022.png,Lung_Opacity/masks/Lung_Opacity-2022.png,True,G04540,test,9d47a153f0cd6b4081b5c3b5238fe4f1,8168545e5a7f217e
Lung_Opacity-2023.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2023.png,Lung_Opacity/masks/Lung_Opacity-2023.png,True,G04541,test,83652a8558e264741af71ac14c8aa8f2,d0bd4e0c4a2f6b4b
Lung_Opacity-2024.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2024.png,Lung_Opacity/masks/Lung_Opacity-2024.png,True,G04542,train,ea38f7701e577821d70384e084cd7a8d,c07ebf67a00f8217
Lung_Opacity-2025.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2025.png,Lung_Opacity/masks/Lung_Opacity-2025.png,True,G04543,test,4e1c157faed963ae6903c7c74739ce03,c82ad3555e555c5c
Lung_Opacity-2026.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2026.png,Lung_Opacity/masks/Lung_Opacity-2026.png,True,G04544,train,c080caa519a8aa1b8fcb69f066215867,c13d56071f0f1e1b
Lung_Opacity-2027.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2027.png,Lung_Opacity/masks/Lung_Opacity-2027.png,True,G04545,test,991d55b0ff4d63e6e1fd61150cb29eb5,83307b3e38bcb4b4
Lung_Opacity-2028.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2028.png,Lung_Opacity/masks/Lung_Opacity-2028.png,True,G04546,train,3e89d81e810cdc8ac262ed2369e73045,957f02450e4f5c5d
Lung_Opacity-2029.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2029.png,Lung_Opacity/masks/Lung_Opacity-2029.png,True,G04547,train,91f9b33560c7226a41b8b71ed5528281,d87f9652092d3c5c
Lung_Opacity-203.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-203.png,Lung_Opacity/masks/Lung_Opacity-203.png,True,G04548,train,fedede45b58d5425b88d410e9eb0ab93,81681e161f1f595f
Lung_Opacity-2030.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2030.png,Lung_Opacity/masks/Lung_Opacity-2030.png,True,G04549,train,fb44e11abf8b5f594e92db14c9cedbbb,ec5d430d0c0f8f8f
Lung_Opacity-2031.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2031.png,Lung_Opacity/masks/Lung_Opacity-2031.png,True,G04550,val,7d8efde94d22ff60e45cdd8bde1e0bfc,c17957d2ca4b5c1c
Lung_Opacity-2032.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2032.png,Lung_Opacity/masks/Lung_Opacity-2032.png,True,G04551,val,c71b68895c914e7b1f78a6ce94de67c0,8179165e5a78f8e8
Lung_Opacity-2033.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2033.png,Lung_Opacity/masks/Lung_Opacity-2033.png,True,G04552,train,6ae9d0d537a5a2416eef9c0387b8d21d,d12bd6125f58585e
Lung_Opacity-2034.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2034.png,Lung_Opacity/masks/Lung_Opacity-2034.png,True,G04553,val,3cdf9a9ac8b8758dc304368599c6722b,894c0e5e4be3e786
Lung_Opacity-2035.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2035.png,Lung_Opacity/masks/Lung_Opacity-2035.png,True,G04554,test,88af298222ad6da4dd8ea46ad8cfb16a,893d965a5979caa1
Lung_Opacity-2036.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2036.png,Lung_Opacity/masks/Lung_Opacity-2036.png,True,G04555,test,4a9a1528cc2dd88be9be48841cb4445e,d879c69e461a39ca
Lung_Opacity-2037.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2037.png,Lung_Opacity/masks/Lung_Opacity-2037.png,True,G04556,train,83dd786cf7334617f8b0ac0f5685494a,d47d434a0f0f4d4d
Lung_Opacity-2038.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2038.png,Lung_Opacity/masks/Lung_Opacity-2038.png,True,G04557,train,e077e7008912329d88e547b313fa31b3,d939c61b1a5b1d19
Lung_Opacity-2039.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2039.png,Lung_Opacity/masks/Lung_Opacity-2039.png,True,G04558,train,263a4ed210fe7f603105c14b9cc6b752,8178045f797969e5
Lung_Opacity-204.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-204.png,Lung_Opacity/masks/Lung_Opacity-204.png,True,G04559,train,61a7e97c9592b37c63a468b1fb0e49ae,894d865e487b69e5
Lung_Opacity-2040.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2040.png,Lung_Opacity/masks/Lung_Opacity-2040.png,True,G04560,train,1df9f520686dc9fe28893e35971a27c5,8129c446125fdf3f
Lung_Opacity-2041.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2041.png,Lung_Opacity/masks/Lung_Opacity-2041.png,True,G04561,train,e9fcda47ce79e2a8b7d99c14ee5be3bf,9178411b5b4f3d5c
Lung_Opacity-2042.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2042.png,Lung_Opacity/masks/Lung_Opacity-2042.png,True,G04562,train,00b475dff0092fe8ecce6514ad87aeb5,c97f46061f1b0a3b
Lung_Opacity-2043.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2043.png,Lung_Opacity/masks/Lung_Opacity-2043.png,True,G04563,train,d9daab9085508ed54d4b73c940937171,c42e0f0f4d7c6c6c
Lung_Opacity-2044.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2044.png,Lung_Opacity/masks/Lung_Opacity-2044.png,True,G04564,train,a21eeb1f8fb4c8de362d0fb4a41aea9f,9c2f831a0d1e5ee3
Lung_Opacity-2045.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2045.png,Lung_Opacity/masks/Lung_Opacity-2045.png,True,G04565,train,4d76046b5fbee3d21d3214cf86bdf718,c13b7e0a5e287939
Lung_Opacity-2046.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2046.png,Lung_Opacity/masks/Lung_Opacity-2046.png,True,G04566,train,b7f7d265682385e19bd34d107c03252a,d8595b1a5a0b1f2e
Lung_Opacity-2047.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2047.png,Lung_Opacity/masks/Lung_Opacity-2047.png,True,G04567,train,06ffb0826dff1e57fb469f9cf4efa9d6,d8195b1b39b4f4a4
Lung_Opacity-2048.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2048.png,Lung_Opacity/masks/Lung_Opacity-2048.png,True,G04568,train,fbcd7db750c14f735a5cc2eb02d87eec,8370155d5b5b583b
Lung_Opacity-2049.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2049.png,Lung_Opacity/masks/Lung_Opacity-2049.png,True,G04569,train,75ef636d1c34ed2aa40b910ded7254f3,99211e5f1a68f9a9
Lung_Opacity-205.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-205.png,Lung_Opacity/masks/Lung_Opacity-205.png,True,G04570,train,81d0168a634ee2fb993bf61b30d0b1cc,996c0e5b697cc6a2
Lung_Opacity-2050.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2050.png,Lung_Opacity/masks/Lung_Opacity-2050.png,True,G04571,train,1c370ab32373190aabb0aad97f04ee82,887f161a1d584b5f
Lung_Opacity-2051.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2051.png,Lung_Opacity/masks/Lung_Opacity-2051.png,True,G04572,train,f1c8ff949c5c2213d4d2c8e9921c7305,842c57cb4b3e4c6e
Lung_Opacity-2052.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2052.png,Lung_Opacity/masks/Lung_Opacity-2052.png,True,G04573,train,6b2255b68944c47d89d4dbf710963701,8879421b1b395f1f
Lung_Opacity-2053.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2053.png,Lung_Opacity/masks/Lung_Opacity-2053.png,True,G04574,train,f918547152919de9a9b178a1077006f6,c03d421f1a1b797b
Lung_Opacity-2054.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2054.png,Lung_Opacity/masks/Lung_Opacity-2054.png,True,G04575,train,914abd75c8028fb8e8871c859eb4b83e,d42e551f1f4c0f0e
Lung_Opacity-2055.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2055.png,Lung_Opacity/masks/Lung_Opacity-2055.png,True,G04576,train,5e9226f907710329eaf4dee5071b2ea1,cc2e534b0d0eadad
Lung_Opacity-2056.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2056.png,Lung_Opacity/masks/Lung_Opacity-2056.png,True,G04577,val,e57f63e64196c8df676efbaabd724d55,dc395b0d1c181f9d
Lung_Opacity-2057.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2057.png,Lung_Opacity/masks/Lung_Opacity-2057.png,True,G04578,val,a4472f5326ee4359818382c6c0aa51e4,90384b451d1fe53f
Lung_Opacity-2058.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2058.png,Lung_Opacity/masks/Lung_Opacity-2058.png,True,G04579,test,1a4308241170138f60a2d8ccee4fc367,817fc2571d45455d
Lung_Opacity-2059.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2059.png,Lung_Opacity/masks/Lung_Opacity-2059.png,True,G04580,val,b7fb38c2ef965d5f941754b3c06b8518,896887171b871f3b
Lung_Opacity-206.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-206.png,Lung_Opacity/masks/Lung_Opacity-206.png,True,G04581,train,c7435b613873f2a062f40ef95ea7b585,8829061f5e5f5a5b
Lung_Opacity-2060.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2060.png,Lung_Opacity/masks/Lung_Opacity-2060.png,True,G04582,test,1fa0c09d3613014dbce03ae25c7ebff3,817bf6d65a514943
Lung_Opacity-2061.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2061.png,Lung_Opacity/masks/Lung_Opacity-2061.png,True,G04583,test,a310baa52d3787707667736d227efc4f,83780f171f587a5a
Lung_Opacity-2062.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2062.png,Lung_Opacity/masks/Lung_Opacity-2062.png,True,G04584,train,3fa6f4ae1c7fc2089de9a25f5812cb80,dc2fc30f0e490dad
Lung_Opacity-2063.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2063.png,Lung_Opacity/masks/Lung_Opacity-2063.png,True,G04585,train,34b12b58712f3129231f8aaaa71d42fb,917947561e4b4f1c
Lung_Opacity-2064.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2064.png,Lung_Opacity/masks/Lung_Opacity-2064.png,True,G04586,val,1962735c06749d6ba7fe010c81c9e8e8,c13d5d824d195add
Lung_Opacity-2065.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2065.png,Lung_Opacity/masks/Lung_Opacity-2065.png,True,G04587,train,ff0ae78fba6f377393df582d3c6d332a,887e060a5f6c78f9
Lung_Opacity-2066.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2066.png,Lung_Opacity/masks/Lung_Opacity-2066.png,True,G04588,train,74612973bb32ee9b317dbe55a25091f1,c81d134b1d38b9eb
Lung_Opacity-2067.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2067.png,Lung_Opacity/masks/Lung_Opacity-2067.png,True,G04589,val,0e4f08b4167362703d3d4f9ab7cc2155,952b4a1f190a7adb
Lung_Opacity-2068.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2068.png,Lung_Opacity/masks/Lung_Opacity-2068.png,True,G04590,test,c9a17ce7f1cc88fed2cc127c2f81bc42,c02d56921f5f4b4e
Lung_Opacity-2069.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2069.png,Lung_Opacity/masks/Lung_Opacity-2069.png,True,G04591,train,15667516d28a172bc4ee78d12349ec81,917cc61a196d3e3c
Lung_Opacity-207.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-207.png,Lung_Opacity/masks/Lung_Opacity-207.png,True,G04592,test,a7196466ed3f72080e05ad7114ee2939,c03dd34b4c687c3e
Lung_Opacity-2070.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2070.png,Lung_Opacity/masks/Lung_Opacity-2070.png,True,G04593,train,8e4fddcba494bea3017670210aad34d2,817886160b5f2f7d
Lung_Opacity-2071.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2071.png,Lung_Opacity/masks/Lung_Opacity-2071.png,True,G04594,train,bbcd3322bd5252a2cb7caf94736c0141,c93bd7161a19691b
Lung_Opacity-2072.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2072.png,Lung_Opacity/masks/Lung_Opacity-2072.png,True,G04595,train,38840214c171a2b455b2880f337b1d0d,856a44564b2fa6e7
Lung_Opacity-2073.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2073.png,Lung_Opacity/masks/Lung_Opacity-2073.png,True,G04596,train,2b39d7e16675036dda505d537ab234f9,d63d6949454b2a2f
Lung_Opacity-2074.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2074.png,Lung_Opacity/masks/Lung_Opacity-2074.png,True,G04597,train,a2ea9cb0be053a992f9ab8bb5a04fdf2,917f86121b1f185f
Lung_Opacity-2075.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2075.png,Lung_Opacity/masks/Lung_Opacity-2075.png,True,G04598,train,34fafc48a643ad0ba8c8de6c98c4172e,dd2a51435f0b4c2f
Lung_Opacity-2076.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2076.png,Lung_Opacity/masks/Lung_Opacity-2076.png,True,G04599,train,777498f758c2ab3b0f6f357ac3a39e03,dc2dc21b1f4c1d5c
Lung_Opacity-2077.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2077.png,Lung_Opacity/masks/Lung_Opacity-2077.png,True,G04600,train,57a687afc7a7307f13de30fb1eaca5c5,c07d6d481c5a4d7d
Lung_Opacity-2078.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2078.png,Lung_Opacity/masks/Lung_Opacity-2078.png,True,G04601,val,4dba32b27d8f725cdb16be6b5ca31a19,d87d424f4e4f0f0a
Lung_Opacity-2079.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2079.png,Lung_Opacity/masks/Lung_Opacity-2079.png,True,G04602,test,9aaeaae696581deeada5b3e37b9f011e,95936b9971538363
Lung_Opacity-208.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-208.png,Lung_Opacity/masks/Lung_Opacity-208.png,True,G04603,train,f727386f750edf39ce103b677ba41e11,d82dc30f1aab38ab
Lung_Opacity-2080.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2080.png,Lung_Opacity/masks/Lung_Opacity-2080.png,True,G04604,train,18f9fb16b4134616f29e7bbf149d8f2c,897c045a797d4be1
Lung_Opacity-2081.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2081.png,Lung_Opacity/masks/Lung_Opacity-2081.png,True,G04605,train,3b1116565a7b9df44c0c66ff4d66fce2,cc35db4f5c08781e
Lung_Opacity-2082.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2082.png,Lung_Opacity/masks/Lung_Opacity-2082.png,True,G04606,train,9205eccafc2a9d0fe9a0fcd0dba19777,d02b4f38493939bb
Lung_Opacity-2083.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2083.png,Lung_Opacity/masks/Lung_Opacity-2083.png,True,G04607,train,0a4cdbfe74430e2bdcbd811667b2185a,856e065e486f6c6d
Lung_Opacity-2084.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2084.png,Lung_Opacity/masks/Lung_Opacity-2084.png,True,G04608,train,5e1546a14481a3c2529c26e575cbbb06,837a141e5a5b515f
Lung_Opacity-2085.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2085.png,Lung_Opacity/masks/Lung_Opacity-2085.png,True,G04609,train,5f8ed1f71083adfcce0b181a272013ee,d93bd65a181a195b
Lung_Opacity-2086.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2086.png,Lung_Opacity/masks/Lung_Opacity-2086.png,True,G04610,train,86dd2f36f0c4e4d9b52d7b5ae298c5e0,d52fd2c14c4e4e1e
Lung_Opacity-2087.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2087.png,Lung_Opacity/masks/Lung_Opacity-2087.png,True,G04611,train,076779ebc6bc91c647bccd541c304b87,856dc84a4d4eccde
Lung_Opacity-2088.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2088.png,Lung_Opacity/masks/Lung_Opacity-2088.png,True,G04612,train,013909a8b78e8167205d4f0170b7b141,cd2cb61ee1c21af1
Lung_Opacity-2089.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2089.png,Lung_Opacity/masks/Lung_Opacity-2089.png,True,G04613,train,1903d7ef12dcae807999ae30eb67a0e7,8178c7161e4f5b59
Lung_Opacity-209.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-209.png,Lung_Opacity/masks/Lung_Opacity-209.png,True,G04614,train,22a6c451ef380b3e4e7354691bd0303d,857f15540f0f0f0f
Lung_Opacity-2090.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2090.png,Lung_Opacity/masks/Lung_Opacity-2090.png,True,G04615,train,ad36c3b8a094db1e2e4b0a7d70341b7d,836054155f5a7b3b
Lung_Opacity-2091.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2091.png,Lung_Opacity/masks/Lung_Opacity-2091.png,True,G04616,train,960a7c5642d1bedf13254b55665bcfb4,9ef5430b4b0f0f29
Lung_Opacity-2092.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2092.png,Lung_Opacity/masks/Lung_Opacity-2092.png,True,G04617,train,8edfa7c630ef819e970e6887d5e3ab22,913a081f191f5f5b
Lung_Opacity-2093.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2093.png,Lung_Opacity/masks/Lung_Opacity-2093.png,True,G04618,val,b5f8bfb5a1aef7135a86e47cebf437b6,de2d411b6d3a3a23
Lung_Opacity-2094.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2094.png,Lung_Opacity/masks/Lung_Opacity-2094.png,True,G04619,train,a003513d9c2d20477f6e7722a801fe33,d439534b191b3d2d
Lung_Opacity-2095.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2095.png,Lung_Opacity/masks/Lung_Opacity-2095.png,True,G04620,train,6a5b42a3ea825409e61fa72cc458361c,c02d423f3a37371b
Lung_Opacity-2096.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2096.png,Lung_Opacity/masks/Lung_Opacity-2096.png,True,G04621,val,1ca95c50e936c489e80aa577db7fad2c,93785b5c787b4185
Lung_Opacity-2097.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2097.png,Lung_Opacity/masks/Lung_Opacity-2097.png,True,G04622,val,17a652d6f6adf2753bf79d7ff76bad90,9d6b35527868726c
Lung_Opacity-2098.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2098.png,Lung_Opacity/masks/Lung_Opacity-2098.png,True,G04623,test,5748889d4954ae60434634acffe71784,996906171f5e791a
Lung_Opacity-2099.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2099.png,Lung_Opacity/masks/Lung_Opacity-2099.png,True,G04624,test,b01d11c519c09cb1ea08387cc41339c6,d426454a0e2f2f6f
Lung_Opacity-21.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-21.png,Lung_Opacity/masks/Lung_Opacity-21.png,True,G04625,test,475e3e00375ef575594bcdfcd0b675f2,d03d5a0b0a1a3f9f
Lung_Opacity-210.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-210.png,Lung_Opacity/masks/Lung_Opacity-210.png,True,G04626,train,677ae0d6ba99edb529db071e069a80c8,d13b581bf8c6f4c0
Lung_Opacity-2100.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2100.png,Lung_Opacity/masks/Lung_Opacity-2100.png,True,G04627,train,bd10db730933e8cebcd9ea5415e11aee,d02d37c43ec02fce
Lung_Opacity-2101.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2101.png,Lung_Opacity/masks/Lung_Opacity-2101.png,True,G04628,train,da0f06e2c8d33ffb6780dd34e863c6db,d42c438b5f2d4e1d
Lung_Opacity-2102.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2102.png,Lung_Opacity/masks/Lung_Opacity-2102.png,True,G04629,train,73c23bb1071b80956f2098fc0378ea68,817d0616585a7f79
Lung_Opacity-2103.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2103.png,Lung_Opacity/masks/Lung_Opacity-2103.png,True,G04630,train,7eb2b870e587296da8831f6c595ddf81,8979125a59595a7b
Lung_Opacity-2104.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2104.png,Lung_Opacity/masks/Lung_Opacity-2104.png,True,G04631,train,bc000b71fe3c0e005d5a945bbd1e15f0,995fd61a59c506c6
Lung_Opacity-2105.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2105.png,Lung_Opacity/masks/Lung_Opacity-2105.png,True,G04632,test,ab764b03144bb6c155b8384bcfa7e5e2,917b47945ac13b99
Lung_Opacity-2106.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2106.png,Lung_Opacity/masks/Lung_Opacity-2106.png,True,G04633,train,0f3acaa7901363016756fada873b64f9,cc3ed14b424f4c4f
Lung_Opacity-2107.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2107.png,Lung_Opacity/masks/Lung_Opacity-2107.png,True,G04634,train,9532ca26262c9f32aec08794e862ee81,d03f471a085f1f1b
Lung_Opacity-2108.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2108.png,Lung_Opacity/masks/Lung_Opacity-2108.png,True,G04635,train,6c963536e224cefbee067a23f6016dba,d42fc34d4e434f0e
Lung_Opacity-2109.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2109.png,Lung_Opacity/masks/Lung_Opacity-2109.png,True,G04636,train,e4d56dc68617b2ef03865ac8408f5a32,817d8f161e0b4d5d
Lung_Opacity-211.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-211.png,Lung_Opacity/masks/Lung_Opacity-211.png,True,G04637,train,e171450cbc33200f898db7c7216ca60a,d42e57934f4f0e0c
Lung_Opacity-2110.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2110.png,Lung_Opacity/masks/Lung_Opacity-2110.png,True,G04638,val,0c70cca62a3a64712e1f4875611c6e64,917947161e5b4a5b
Lung_Opacity-2111.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2111.png,Lung_Opacity/masks/Lung_Opacity-2111.png,True,G04639,val,83542c81296a27a594c661dd961338c0,857c860f5f185f49
Lung_Opacity-2112.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2112.png,Lung_Opacity/masks/Lung_Opacity-2112.png,True,G04640,train,cab73da22473c1894bcc5680c24cd7de,956846161b5f1d5d
Lung_Opacity-2113.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2113.png,Lung_Opacity/masks/Lung_Opacity-2113.png,True,G04641,test,f1bb1747f3f07ca4150d390ee298aef6,c01d1a0b5f0efc3d
Lung_Opacity-2114.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2114.png,Lung_Opacity/masks/Lung_Opacity-2114.png,True,G04642,val,d5ffbab4de7a7f7cccd8f3de16d161b0,dc3ad38faea0a4e0
Lung_Opacity-2115.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2115.png,Lung_Opacity/masks/Lung_Opacity-2115.png,True,G04643,test,14dca2b2189330c72e6e9e0f0d342bdd,d42f821bc1c71ecb
Lung_Opacity-2116.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2116.png,Lung_Opacity/masks/Lung_Opacity-2116.png,True,G04644,train,6f872acd83db6c6d0b8cdc93d9e2bdd6,cc3d021b5f0e1acf
Lung_Opacity-2117.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2117.png,Lung_Opacity/masks/Lung_Opacity-2117.png,True,G04645,train,dfa5bb9ec3888a96832d03d292dcd98e,917f061a594978fc
Lung_Opacity-2118.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2118.png,Lung_Opacity/masks/Lung_Opacity-2118.png,True,G04646,train,10a60d40abbf33320b3a50ce23ce6116,8370151f7b307f70
Lung_Opacity-2119.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2119.png,Lung_Opacity/masks/Lung_Opacity-2119.png,True,G04647,train,852bd82ff53053e0582561c53b8c954e,81780f1f39772276
Lung_Opacity-212.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-212.png,Lung_Opacity/masks/Lung_Opacity-212.png,True,G04648,train,2acb3cf2832af84a49fd4da86c2b1da4,c43d43cb0ade2dcc
Lung_Opacity-2120.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2120.png,Lung_Opacity/masks/Lung_Opacity-2120.png,True,G04649,train,f315748bd45e5b8b6a8091cd3c2ec36b,cd2e925b4b54434f
Lung_Opacity-2121.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2121.png,Lung_Opacity/masks/Lung_Opacity-2121.png,True,G04650,train,b97c900d3ae228f4913db9e1ed734803,903b471e1d591f5a
Lung_Opacity-2122.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2122.png,Lung_Opacity/masks/Lung_Opacity-2122.png,True,G04651,train,4ad75018eeb4f67717fbd94206e82fce,d42a42d54f0a6f5d
Lung_Opacity-2123.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2123.png,Lung_Opacity/masks/Lung_Opacity-2123.png,True,G04652,train,c88fac98c4bb625eb43c5a60920de649,822f0f0a6f6e6e4a
Lung_Opacity-2124.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2124.png,Lung_Opacity/masks/Lung_Opacity-2124.png,True,G04653,val,a0cea4cef66819d2b601536dae06b45a,997847165ac91d79
Lung_Opacity-2125.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2125.png,Lung_Opacity/masks/Lung_Opacity-2125.png,True,G04654,val,31102267f655251650b68748e85e03d3,8879cec71e58c91b
Lung_Opacity-2126.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2126.png,Lung_Opacity/masks/Lung_Opacity-2126.png,True,G04655,train,ca50fb2b04ce0e7a6c6c85bc70048a59,cc0d4b0e0e2e6dbd
Lung_Opacity-2127.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2127.png,Lung_Opacity/masks/Lung_Opacity-2127.png,True,G04656,test,83e50af8d12f4e528c53292c507102fb,dc2e434f4c4e4a6e
Lung_Opacity-2128.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2128.png,Lung_Opacity/masks/Lung_Opacity-2128.png,True,G04657,train,5f5029fdbc0d4b6d513adde7d874a530,d42cd5c34b4b4b4b
Lung_Opacity-2129.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2129.png,Lung_Opacity/masks/Lung_Opacity-2129.png,True,G04658,train,9bd855d56d23e0d11e481e073b28fcff,a92f869ac9f5e0e0
Lung_Opacity-213.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-213.png,Lung_Opacity/masks/Lung_Opacity-213.png,True,G04659,test,a172cc40274099e561619891270e6f8a,d528af0b0b4d1f1e
Lung_Opacity-2130.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2130.png,Lung_Opacity/masks/Lung_Opacity-2130.png,True,G04660,test,7ee0365eeca27390f40e63c25683f737,d52b5752da5a584a
Lung_Opacity-2131.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2131.png,Lung_Opacity/masks/Lung_Opacity-2131.png,True,G04661,train,e34728291cac0f965901889fda239ffe,917887161a4f7e39
Lung_Opacity-2132.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2132.png,Lung_Opacity/masks/Lung_Opacity-2132.png,True,G04662,train,ec526b13afecfa5665dca8e2c627879c,c12914561f5b197f
Lung_Opacity-2133.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2133.png,Lung_Opacity/masks/Lung_Opacity-2133.png,True,G04663,val,25c47579564a77507fe80a9bf2b2aa74,817b041e597b3b3a
Lung_Opacity-2134.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2134.png,Lung_Opacity/masks/Lung_Opacity-2134.png,True,G04664,train,5cc93876f9d89c03b22e224d442864a9,853c7a4a3ae66267
Lung_Opacity-2135.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2135.png,Lung_Opacity/masks/Lung_Opacity-2135.png,True,G04665,train,b873b5edfcf6cc3b4ca14985456ef689,c07d450a0f5d6c7d
Lung_Opacity-2136.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2136.png,Lung_Opacity/masks/Lung_Opacity-2136.png,True,G04666,train,f8f8980747a313cb70cd5ca54c17b371,c42f570b0f0e0f4f
Lung_Opacity-2137.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2137.png,Lung_Opacity/masks/Lung_Opacity-2137.png,True,G04667,train,a2d1ab6200ed8448511034d5e4c7d64a,93699d185b785d4a
Lung_Opacity-2138.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2138.png,Lung_Opacity/masks/Lung_Opacity-2138.png,True,G04668,train,99e73a6505e589f753b3cd1d43173786,c92d461a1c6d69ed
Lung_Opacity-2139.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2139.png,Lung_Opacity/masks/Lung_Opacity-2139.png,True,G04669,train,f947f185b0c5d6eee4870569e56bf59f,916a1618da79f8f8
Lung_Opacity-214.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-214.png,Lung_Opacity/masks/Lung_Opacity-214.png,True,G04670,train,eba1d83cdad320a650f05711878882df,887f065a19390fbd
Lung_Opacity-2140.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2140.png,Lung_Opacity/masks/Lung_Opacity-2140.png,True,G04671,train,55e5118ea9ca1eca7b64df7d98d19e8b,852e071e0b0f1f3f
Lung_Opacity-2141.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2141.png,Lung_Opacity/masks/Lung_Opacity-2141.png,True,G04672,train,9e037280fc22c0ca68db464c813f55e8,8c2f830b0f0f2d3f
Lung_Opacity-2142.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2142.png,Lung_Opacity/masks/Lung_Opacity-2142.png,True,G04673,train,f86040fa069e829af868730dd6c683e8,9d7dc20f051a1d5e
Lung_Opacity-2143.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2143.png,Lung_Opacity/masks/Lung_Opacity-2143.png,True,G04674,train,bc5fc16b3ac55d4e919a5b9032514967,9c6e474b4f094f19
Lung_Opacity-2144.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2144.png,Lung_Opacity/masks/Lung_Opacity-2144.png,True,G04675,val,d2ba37e464fd2adf4b5ebd5b64f3b4b6,936b4d7c716c6524
Lung_Opacity-2145.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2145.png,Lung_Opacity/masks/Lung_Opacity-2145.png,True,G04676,train,715944efff125ab2f27324f0c48afa4c,c03c4b1f0b0e6d7d
Lung_Opacity-2146.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2146.png,Lung_Opacity/masks/Lung_Opacity-2146.png,True,G04677,train,43d710f7733997f7c856f499c1f12487,d46c070a0f6e6e6e
Lung_Opacity-2147.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2147.png,Lung_Opacity/masks/Lung_Opacity-2147.png,True,G04678,train,85270328ad6606066963ed008d8545e5,d42e07070f0f4f6e
Lung_Opacity-2148.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2148.png,Lung_Opacity/masks/Lung_Opacity-2148.png,True,G04679,val,dc18804ef6a6ad7b01968613253b91e6,cd3ad3085b1a5e1b
Lung_Opacity-2149.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2149.png,Lung_Opacity/masks/Lung_Opacity-2149.png,True,G04680,val,1b16b3bb2ec12cdc913bfb770685aeea,9f690c6e73e02237
Lung_Opacity-215.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-215.png,Lung_Opacity/masks/Lung_Opacity-215.png,True,G04681,test,7cb29030de2737772c45e3104f9eafa4,9168c61a1e7e794d
Lung_Opacity-2150.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2150.png,Lung_Opacity/masks/Lung_Opacity-2150.png,True,G04682,test,63f8faeb00149132923b5e105613f436,887f071a1e4f1b1d
Lung_Opacity-2151.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2151.png,Lung_Opacity/masks/Lung_Opacity-2151.png,True,G04683,test,bfbaf2bf1f48857f5572b55fc68c2a54,dd2ad252db430e1b
Lung_Opacity-2152.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2152.png,Lung_Opacity/masks/Lung_Opacity-2152.png,True,G04684,val,38496687ff29388adfc5021a454855fb,cd1e1a492d0eafa6
Lung_Opacity-2153.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2153.png,Lung_Opacity/masks/Lung_Opacity-2153.png,True,G04685,train,340cee71062750581f8e32841839ce45,817f8e0d4d4f4f44
Lung_Opacity-2154.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2154.png,Lung_Opacity/masks/Lung_Opacity-2154.png,True,G04686,train,b14b4aa31534757275e06807352cd6c7,cd3c164b1e4a5f58
Lung_Opacity-2155.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2155.png,Lung_Opacity/masks/Lung_Opacity-2155.png,True,G04687,val,2ee9742a9aa06d2610d4eb55af879df1,9168575a594d7974
Lung_Opacity-2156.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2156.png,Lung_Opacity/masks/Lung_Opacity-2156.png,True,G04688,test,fd35293f71b94825e8f7a5b691a295f2,8968165a597b7aca
Lung_Opacity-2157.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2157.png,Lung_Opacity/masks/Lung_Opacity-2157.png,True,G04689,train,bb32dfb8a59b0d9d76ee80bab82f6782,d02d5b5b0828babf
Lung_Opacity-2158.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2158.png,Lung_Opacity/masks/Lung_Opacity-2158.png,True,G04690,train,92e9b440189cc23d31198ffd5b438bba,817f8702955e9657
Lung_Opacity-2159.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2159.png,Lung_Opacity/masks/Lung_Opacity-2159.png,True,G04691,train,edc80637ac44518d7eaa053517781e0d,856e860f450f6f0f
Lung_Opacity-216.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-216.png,Lung_Opacity/masks/Lung_Opacity-216.png,True,G04692,train,5551e48dc109b095ee5e57a2f3a62d0e,c139521f386a7d39
Lung_Opacity-2160.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2160.png,Lung_Opacity/masks/Lung_Opacity-2160.png,True,G04693,train,f1b60f1f3b32fe2d60981d13deba97a9,d42e834b4d7c6c4e
Lung_Opacity-2161.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2161.png,Lung_Opacity/masks/Lung_Opacity-2161.png,True,G04694,train,e4d9b9190fde67b15185a8bd23a2efad,914d8c127c3def0e
Lung_Opacity-2162.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2162.png,Lung_Opacity/masks/Lung_Opacity-2162.png,True,G04695,train,74878223b0100ff08bd35cd61e02a246,d42d53574a0b6f0b
Lung_Opacity-2163.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2163.png,Lung_Opacity/masks/Lung_Opacity-2163.png,True,G04696,train,cd62eb72eead12d7bb9107f15d602e7f,d42e070a7c627df1
Lung_Opacity-2164.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2164.png,Lung_Opacity/masks/Lung_Opacity-2164.png,True,G04697,train,a1e11dcffa7c8ac106fbf9d173e34a89,8578878a5929797d
Lung_Opacity-2165.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2165.png,Lung_Opacity/masks/Lung_Opacity-2165.png,True,G04698,train,c8e6795799911127aa3f69bf9bd935c3,9168861e5a5e5e7a
Lung_Opacity-2166.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2166.png,Lung_Opacity/masks/Lung_Opacity-2166.png,True,G04699,train,272c9d9f674c64bb6ec9ce04f3c316e1,c46e87194c8c7bbc
Lung_Opacity-2167.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2167.png,Lung_Opacity/masks/Lung_Opacity-2167.png,True,G04700,train,79cfedd96c0436a0a0e95da3f34a023b,c56e47434b4b4b4b
Lung_Opacity-2168.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2168.png,Lung_Opacity/masks/Lung_Opacity-2168.png,True,G04701,val,3de500dc40e7e7217c93f4117164dacd,9139c397181f6b3a
Lung_Opacity-2169.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2169.png,Lung_Opacity/masks/Lung_Opacity-2169.png,True,G04702,test,8855158fd73890b23c275439d38e22ae,d42d121757035f1f
Lung_Opacity-217.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-217.png,Lung_Opacity/masks/Lung_Opacity-217.png,True,G04703,test,285fb2faab61ab90f9f8eda474afb2d5,c03d5a5d091c3f3d
Lung_Opacity-2170.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2170.png,Lung_Opacity/masks/Lung_Opacity-2170.png,True,G04704,train,990dc79814126e76eb320088504dada5,995e161a587ae9e8
Lung_Opacity-2171.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2171.png,Lung_Opacity/masks/Lung_Opacity-2171.png,True,G04705,train,e500a1d710a7b318c1e01d8ec18d4e8d,dc2ec14b5f0a0d2f
Lung_Opacity-2172.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2172.png,Lung_Opacity/masks/Lung_Opacity-2172.png,True,G04706,train,d0e8a12e10d2672fa7dfa378e1ad9afa,d9290b195d58f8f2
Lung_Opacity-2173.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2173.png,Lung_Opacity/masks/Lung_Opacity-2173.png,True,G04707,train,8d66968efd695faae540f65caeb96e6e,9d62533f58585959
Lung_Opacity-2174.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2174.png,Lung_Opacity/masks/Lung_Opacity-2174.png,True,G04708,train,5dd173bf3883d7bdbe200d940d3741a3,887f05161e5e5b59
Lung_Opacity-2175.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2175.png,Lung_Opacity/masks/Lung_Opacity-2175.png,True,G04709,train,ad743a0ac9a79b100fea0e7ba5bcf444,d46f450a0a2f1e7d
Lung_Opacity-2176.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2176.png,Lung_Opacity/masks/Lung_Opacity-2176.png,True,G04710,train,763a5dc04878a1f38a0a9b9894b7bbaa,913bcf0e1e4e5958
Lung_Opacity-2177.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2177.png,Lung_Opacity/masks/Lung_Opacity-2177.png,True,G04711,train,9878d250ad75358e62aa3efbd00ae8e9,c0357a405bcb2ded
Lung_Opacity-2178.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2178.png,Lung_Opacity/masks/Lung_Opacity-2178.png,True,G04712,train,533e8d2bb1c93af8d195c8f36ffc9bf3,d42bc31d5f484c5e
Lung_Opacity-2179.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2179.png,Lung_Opacity/masks/Lung_Opacity-2179.png,True,G04713,train,db08f6ceafd21490b7c5082e4aa792ec,d47fa3558d462d0c
Lung_Opacity-218.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-218.png,Lung_Opacity/masks/Lung_Opacity-218.png,True,G04714,train,8b13369413d03cfc5e3019e2bc73b5ae,d52a07570e1f1c4f
Lung_Opacity-2180.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2180.png,Lung_Opacity/masks/Lung_Opacity-2180.png,True,G04715,train,7dda8023fbd757dfe170fc210cd56398,d0295a1e4f191bbd
Lung_Opacity-2181.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2181.png,Lung_Opacity/masks/Lung_Opacity-2181.png,True,G04716,train,7dfbfa1091b36497283366e48f429f64,802e871f1f1f1f0d
Lung_Opacity-2182.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2182.png,Lung_Opacity/masks/Lung_Opacity-2182.png,True,G04717,val,d2925cd7d24728c7b1968ccac74454d5,d03d4e0c2f2f2ca7
Lung_Opacity-2183.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2183.png,Lung_Opacity/masks/Lung_Opacity-2183.png,True,G04718,train,80217732b9b378bc653214b815e04313,d938165b1d69293d
Lung_Opacity-2184.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2184.png,Lung_Opacity/masks/Lung_Opacity-2184.png,True,G04719,val,210fc6577c7148192f5f65f2e88d2afc,887a131b0d2dbfb2
Lung_Opacity-2185.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2185.png,Lung_Opacity/masks/Lung_Opacity-2185.png,True,G04720,val,9b898a07735eec8ec98a383710a4e572,803f474e5c49697d
Lung_Opacity-2186.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2186.png,Lung_Opacity/masks/Lung_Opacity-2186.png,True,G04721,train,446e5b4f817bbda9c35677b800780e54,917943471e1fcc39
Lung_Opacity-2187.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2187.png,Lung_Opacity/masks/Lung_Opacity-2187.png,True,G04722,test,4a067175f445d67eb43cd4f071eb1a8e,dd2a53098c6abde2
Lung_Opacity-2188.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2188.png,Lung_Opacity/masks/Lung_Opacity-2188.png,True,G04723,val,0a5fdbf309fc62001bb6c4d2dc9e61ab,9879469e1a183fa7
Lung_Opacity-2189.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2189.png,Lung_Opacity/masks/Lung_Opacity-2189.png,True,G04724,val,72b4457d89f77514b7bf40afe2e29bc0,d42f474b0e2e2d0f
Lung_Opacity-219.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-219.png,Lung_Opacity/masks/Lung_Opacity-219.png,True,G04725,test,3d30572e6bebc5694e33d8d64158346e,89a9b77a3a4a1a1b
Lung_Opacity-2190.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2190.png,Lung_Opacity/masks/Lung_Opacity-2190.png,True,G04726,train,7bfc1722f8a9b8052d71ab1653770dc4,956a68555d09795d
Lung_Opacity-2191.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2191.png,Lung_Opacity/masks/Lung_Opacity-2191.png,True,G04727,test,6a73bf0178872d3e41ec59dfbd398b97,91790a5b5b595959
Lung_Opacity-2192.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2192.png,Lung_Opacity/masks/Lung_Opacity-2192.png,True,G04728,train,cce31429d6d74357d78900574977e04a,a82a024b4d7e7a7e
Lung_Opacity-2193.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2193.png,Lung_Opacity/masks/Lung_Opacity-2193.png,True,G04729,test,d0f04078cd79601ba6cc3ec82ff6f129,d52a144a4f6b6b2d
Lung_Opacity-2194.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2194.png,Lung_Opacity/masks/Lung_Opacity-2194.png,True,G04730,val,81a280438d5c3a64f7db6cba77098b01,d02bed420f4e6e4e
Lung_Opacity-2195.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2195.png,Lung_Opacity/masks/Lung_Opacity-2195.png,True,G04731,train,6c92486825784158c517c10f8200ef5c,d91956c60b1f1e4b
Lung_Opacity-2196.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2196.png,Lung_Opacity/masks/Lung_Opacity-2196.png,True,G04732,train,c7887d3e91b1a7a1b298da187524bb5a,992b061e5e1a7ae9
Lung_Opacity-2197.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2197.png,Lung_Opacity/masks/Lung_Opacity-2197.png,True,G04733,train,e4eed336acc80222055ffdd63152349c,9c2bd20f425f19f2
Lung_Opacity-2198.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2198.png,Lung_Opacity/masks/Lung_Opacity-2198.png,True,G04734,train,5b1ca269a3760dd13422229a42588d39,d62fd14b2e6a4d48
Lung_Opacity-2199.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2199.png,Lung_Opacity/masks/Lung_Opacity-2199.png,True,G04735,train,7094624b8926d92ad46b54dde2f46419,dc4ecf0c242f1f83
Lung_Opacity-22.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-22.png,Lung_Opacity/masks/Lung_Opacity-22.png,True,G04736,train,dea8e513d2e337992bddc39cc963aa02,d03b435b1979591b
Lung_Opacity-220.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-220.png,Lung_Opacity/masks/Lung_Opacity-220.png,True,G04737,train,2470f3c188576a7c252758410ee954b5,843c47cb4f4f4d0d
Lung_Opacity-2200.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2200.png,Lung_Opacity/masks/Lung_Opacity-2200.png,True,G04738,train,969748718bd394e1b3b77aafa023bdf2,996b3866a6e42c6d
Lung_Opacity-2201.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2201.png,Lung_Opacity/masks/Lung_Opacity-2201.png,True,G04739,test,41680e29a31582ee5fd4a963d26485e6,d13f42071a1f6b1e
Lung_Opacity-2202.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2202.png,Lung_Opacity/masks/Lung_Opacity-2202.png,True,G04740,train,6a705d32060914068cb4ae06f8950443,d02dc712ca47ff42
Lung_Opacity-2203.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2203.png,Lung_Opacity/masks/Lung_Opacity-2203.png,True,G04741,train,d199bb14421aea5548849e5091c6bd1b,987b45961e5b3998
Lung_Opacity-2204.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2204.png,Lung_Opacity/masks/Lung_Opacity-2204.png,True,G04742,train,8451e1d82e65f1f6d0068bce19953315,9128861e7b4f7a59
Lung_Opacity-2205.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2205.png,Lung_Opacity/masks/Lung_Opacity-2205.png,True,G04743,val,87a4e8f39d8ad1916ce52be8dda63204,d03d471a4b4d2e4f
Lung_Opacity-2206.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2206.png,Lung_Opacity/masks/Lung_Opacity-2206.png,True,G04744,test,a425903efdb03207ce985ca1db4f9148,d02bd31b1b4b1d2d
Lung_Opacity-2207.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2207.png,Lung_Opacity/masks/Lung_Opacity-2207.png,True,G04745,test,672476f55a8815db3c9a623dce961d9e,903f46575a0d4f4d
Lung_Opacity-2208.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2208.png,Lung_Opacity/masks/Lung_Opacity-2208.png,True,G04746,train,3b741bbb9c3bc5d064d541ba47ef9031,c1aa8211f56fb54d
Lung_Opacity-2209.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2209.png,Lung_Opacity/masks/Lung_Opacity-2209.png,True,G04747,train,a09573704400adcab7eb77fdf5e7c5a7,cd2e124b0d5c787d
Lung_Opacity-221.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-221.png,Lung_Opacity/masks/Lung_Opacity-221.png,True,G04748,train,027e427982b7d6dc1c9a261f73147c4c,95285fe963b13332
Lung_Opacity-2210.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2210.png,Lung_Opacity/masks/Lung_Opacity-2210.png,True,G04749,train,dd643f833466baa315fe2c46782e0d70,8179161e5b1b5e59
Lung_Opacity-2211.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2211.png,Lung_Opacity/masks/Lung_Opacity-2211.png,True,G04750,train,f8ecbac505b2a8ad750a9fd52bfc992b,905f4e180f39a9bd
Lung_Opacity-2212.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2212.png,Lung_Opacity/masks/Lung_Opacity-2212.png,True,G04751,train,6178dc9425c8e87163c5e1e257fed7e8,9168961b5b5a395b
Lung_Opacity-2213.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2213.png,Lung_Opacity/masks/Lung_Opacity-2213.png,True,G04752,train,eefbdfa1c0445727de1c2cc068e76823,d52dc25b4b0f4a0f
Lung_Opacity-2214.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2214.png,Lung_Opacity/masks/Lung_Opacity-2214.png,True,G04753,train,e82a18b84ca2fa272f89aab78636c1e9,cc3d5b594a4a6a2e
Lung_Opacity-2215.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2215.png,Lung_Opacity/masks/Lung_Opacity-2215.png,True,G04754,train,e744fd5d7cabb2f9bc7a0b738f80baa0,853b1618591f595f
Lung_Opacity-2216.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2216.png,Lung_Opacity/masks/Lung_Opacity-2216.png,True,G04755,train,cbaae867c8c58daefb999b04381af79d,9178461619497f5f
Lung_Opacity-2217.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2217.png,Lung_Opacity/masks/Lung_Opacity-2217.png,True,G04756,train,1abc502bd4ff6cb6529c2bbfab3354e7,9320845e5d787b79
Lung_Opacity-2218.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2218.png,Lung_Opacity/masks/Lung_Opacity-2218.png,True,G04757,val,f753c2fa91535c02d23c7086c587c7f5,dc3d410f0e4f1c3d
Lung_Opacity-2219.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2219.png,Lung_Opacity/masks/Lung_Opacity-2219.png,True,G04758,train,6b69f2c3adee53119e1f6ff35b187cec,817f94181b1b1b5f
Lung_Opacity-222.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-222.png,Lung_Opacity/masks/Lung_Opacity-222.png,True,G04759,train,202cfd01fbcfe5b447a383bd2d815b46,813b177e7a686863
Lung_Opacity-2220.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2220.png,Lung_Opacity/masks/Lung_Opacity-2220.png,True,G04760,train,546a081ea616469394a243988cc33722,9939bd85161e1e1d
Lung_Opacity-2221.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2221.png,Lung_Opacity/masks/Lung_Opacity-2221.png,True,G04761,val,91af50cb86b5a6bcee3621b35e5e51f5,81784f5c1d2d97a6
Lung_Opacity-2222.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2222.png,Lung_Opacity/masks/Lung_Opacity-2222.png,True,G04762,val,1b47eecd7caf692b598f84d4e379df03,80737d7078606f6b
Lung_Opacity-2223.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2223.png,Lung_Opacity/masks/Lung_Opacity-2223.png,True,G04763,test,c9d98cc7f2d0c14dabfb0383683c6611,c42e054d2a7b3b3b
Lung_Opacity-2224.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2224.png,Lung_Opacity/masks/Lung_Opacity-2224.png,True,G04764,test,49b0ddf1754061137561c05288803e84,d53ba02b9c252d3d
Lung_Opacity-2225.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2225.png,Lung_Opacity/masks/Lung_Opacity-2225.png,True,G04765,test,4f5ecda1eb2825541270945ad006beec,c87cc31b0d3c3b35
Lung_Opacity-2226.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2226.png,Lung_Opacity/masks/Lung_Opacity-2226.png,True,G04766,train,219b04bef51d4dd36e01cefffd85d914,957c1717096f470e
Lung_Opacity-2227.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2227.png,Lung_Opacity/masks/Lung_Opacity-2227.png,True,G04767,train,b84e89c40dfc0d6fbb7aa0fc3cf488d9,d83f434b1d0a2f3c
Lung_Opacity-2228.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2228.png,Lung_Opacity/masks/Lung_Opacity-2228.png,True,G04768,train,b56df2faf8663ad534ba6a47da03b42f,d17b46065e425f5a
Lung_Opacity-2229.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2229.png,Lung_Opacity/masks/Lung_Opacity-2229.png,True,G04769,train,1f84e2bb6d11a8f3c4ac5ea3ef4b1fcc,91694e19194f7d78
Lung_Opacity-223.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-223.png,Lung_Opacity/masks/Lung_Opacity-223.png,True,G04770,train,3df23a308274cd8fea04aa80ee48a775,d52a530f5c0f0f0f
Lung_Opacity-2230.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2230.png,Lung_Opacity/masks/Lung_Opacity-2230.png,True,G04771,val,93a8d9bca6503497d70f7323075d8481,dc3d42431b0e4f3d
Lung_Opacity-2231.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2231.png,Lung_Opacity/masks/Lung_Opacity-2231.png,True,G04772,train,c974808f75e0a99b5d2e17daab5ff3cf,80e88e1f1e5d4d5d
Lung_Opacity-2232.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2232.png,Lung_Opacity/masks/Lung_Opacity-2232.png,True,G04773,train,e9af2545ada3b70e3f83d56233e2bcf3,d42d93435f4b0f4a
Lung_Opacity-2233.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2233.png,Lung_Opacity/masks/Lung_Opacity-2233.png,True,G04774,train,295591ada3d8436b3e2b844a08370079,917a071e1a5e5b78
Lung_Opacity-2234.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2234.png,Lung_Opacity/masks/Lung_Opacity-2234.png,True,G04775,train,b3a9d09884ce6fee8ceae8698fcba939,d03d454b4f4a4f4e
Lung_Opacity-2235.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2235.png,Lung_Opacity/masks/Lung_Opacity-2235.png,True,G04776,train,9e52b2ebd13decd950177b3aab0cb067,896c0e5acf78cac3
Lung_Opacity-2236.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2236.png,Lung_Opacity/masks/Lung_Opacity-2236.png,True,G04777,val,b0ae77e82cda3611b13b63a8c77f53d4,cd2a13191a693dbd
Lung_Opacity-2237.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2237.png,Lung_Opacity/masks/Lung_Opacity-2237.png,True,G04778,train,86217b8880ca00fbb86b5ed8cb5f2636,93281d5f5a4dccc5
Lung_Opacity-2238.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2238.png,Lung_Opacity/masks/Lung_Opacity-2238.png,True,G04779,train,0f43425896925d7ec80d4840de5f86be,c931d21a5a4b7a6b
Lung_Opacity-2239.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2239.png,Lung_Opacity/masks/Lung_Opacity-2239.png,True,G04780,train,8e093e2abe042cc3f837573b4ba4edfb,9139cf1a1e8fb858
Lung_Opacity-224.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-224.png,Lung_Opacity/masks/Lung_Opacity-224.png,True,G04781,train,b4792cda6eb19a18c312100ec92e25df,d52a0f4e0b5d2c6e
Lung_Opacity-2240.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2240.png,Lung_Opacity/masks/Lung_Opacity-2240.png,True,G04782,test,44994d41d4225abcdcdb6c135c010c7d,d829561a1d5938fd
Lung_Opacity-2241.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2241.png,Lung_Opacity/masks/Lung_Opacity-2241.png,True,G04783,train,61a8317a3150a90c0403017bd10742a2,817b525b59393987
Lung_Opacity-2242.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2242.png,Lung_Opacity/masks/Lung_Opacity-2242.png,True,G04784,val,9eca086907b3c8c0f068dc70d195144d,907fc6b9e6e3a005
Lung_Opacity-2243.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2243.png,Lung_Opacity/masks/Lung_Opacity-2243.png,True,G04785,test,93c638cff812a1575455a9e41e99885c,c93e064c7f696c68
Lung_Opacity-2244.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2244.png,Lung_Opacity/masks/Lung_Opacity-2244.png,True,G04786,test,932fb7acd85150c722d44937c9e11100,9178561a693b3937
Lung_Opacity-2245.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2245.png,Lung_Opacity/masks/Lung_Opacity-2245.png,True,G04787,train,6c474e30d1b9eacc7b7954d527b9d070,91281f5e1a5b5a79
Lung_Opacity-2246.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2246.png,Lung_Opacity/masks/Lung_Opacity-2246.png,True,G04788,train,7087ffca7396133c7ddee42449cc4a7e,80794a1f1b193f3d
Lung_Opacity-2247.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2247.png,Lung_Opacity/masks/Lung_Opacity-2247.png,True,G04789,train,4d161d5ee43434e9d205257570291612,895f965a18393f0d
Lung_Opacity-2248.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2248.png,Lung_Opacity/masks/Lung_Opacity-2248.png,True,G04790,train,aacf28717b6afe9779b1bd80e7b44c8e,95288d0d4d3d6dc7
Lung_Opacity-2249.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2249.png,Lung_Opacity/masks/Lung_Opacity-2249.png,True,G04791,train,006f8f5cbb6f886698fee344f268e641,956805061f4f1f5f
Lung_Opacity-225.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-225.png,Lung_Opacity/masks/Lung_Opacity-225.png,True,G04792,train,4c5184e51bbc9aeb3539981b37d7885f,817d87161e171f1d
Lung_Opacity-2250.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2250.png,Lung_Opacity/masks/Lung_Opacity-2250.png,True,G04793,train,c6f93a6a4f7e41f21812e90edb8af6be,dd3c41c30b4b4d4f
Lung_Opacity-2251.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2251.png,Lung_Opacity/masks/Lung_Opacity-2251.png,True,G04794,train,498ee91c3cbe43383d8191242c9f05c1,d036c3c9094f6f2e
Lung_Opacity-2252.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2252.png,Lung_Opacity/masks/Lung_Opacity-2252.png,True,G04795,train,e86b23b5f7fa5658cb022648da36494b,9979c7161e185b1b
Lung_Opacity-2253.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2253.png,Lung_Opacity/masks/Lung_Opacity-2253.png,True,G04796,test,e10083933ca79a06d323ed7f2e6d8be9,9879521b39399e96
Lung_Opacity-2254.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2254.png,Lung_Opacity/masks/Lung_Opacity-2254.png,True,G04797,train,27a1e4b49b17185dbe9d5bd22e4d8fe8,a979861e181f385f
Lung_Opacity-2255.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2255.png,Lung_Opacity/masks/Lung_Opacity-2255.png,True,G04798,train,475e63c70475487b179a397f000b5cd6,993d46165a4f1ad9
Lung_Opacity-2256.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2256.png,Lung_Opacity/masks/Lung_Opacity-2256.png,True,G04799,train,a403a0b28aaa627ffd21bb2152e03d88,916a06185f4f397b
Lung_Opacity-2257.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2257.png,Lung_Opacity/masks/Lung_Opacity-2257.png,True,G04800,train,2de4ef99c5730a584d2ed95cb796c23d,dd2d42124f4a3d3b
Lung_Opacity-2258.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2258.png,Lung_Opacity/masks/Lung_Opacity-2258.png,True,G04801,train,b5e3ccd67a7c4cec4396007485354472,917b400e5a5b7d39
Lung_Opacity-2259.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2259.png,Lung_Opacity/masks/Lung_Opacity-2259.png,True,G04802,val,6a9b83c3a3a47e467df9e244505c995a,994f061a4979697b
Lung_Opacity-226.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-226.png,Lung_Opacity/masks/Lung_Opacity-226.png,True,G04803,val,7da062169a3eebb5a10cc72410771a26,8132161f787e1f3a
Lung_Opacity-2260.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2260.png,Lung_Opacity/masks/Lung_Opacity-2260.png,True,G04804,val,373fb9891ad0ac8378ac1871c7cea390,d42d4b4f4e2e2c2e
Lung_Opacity-2261.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2261.png,Lung_Opacity/masks/Lung_Opacity-2261.png,True,G04805,test,2b276c35c238f812935837b8a364d306,97695e3aa1e74606
Lung_Opacity-2262.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2262.png,Lung_Opacity/masks/Lung_Opacity-2262.png,True,G04806,test,abb6af70299d50b51a7068711cfa1a49,dc2f4d4c4e2e0ad3
Lung_Opacity-2263.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2263.png,Lung_Opacity/masks/Lung_Opacity-2263.png,True,G04807,test,616d9edc12d8b33369cc82abbd913320,917847161e575a3e
Lung_Opacity-2264.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2264.png,Lung_Opacity/masks/Lung_Opacity-2264.png,True,G04808,train,f51271f8561c00e388ac095d5f7aab16,d03d430f1a4e6f3c
Lung_Opacity-2265.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2265.png,Lung_Opacity/masks/Lung_Opacity-2265.png,True,G04809,train,c3799450a5033301bd386b1cadc732bd,c43d13431e7e4b59
Lung_Opacity-2266.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2266.png,Lung_Opacity/masks/Lung_Opacity-2266.png,True,G04810,train,117d01c2c211b6a60c985c7eef799b55,d9195a1b083ab2ef
Lung_Opacity-2267.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2267.png,Lung_Opacity/masks/Lung_Opacity-2267.png,True,G04811,train,109321e701de2da39d3fc69589e0d607,85785a0b2d3db596
Lung_Opacity-2268.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2268.png,Lung_Opacity/masks/Lung_Opacity-2268.png,True,G04812,train,51e65577488dc8c3e4dead830f5b374e,857e47471d4d4c5c
Lung_Opacity-2269.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2269.png,Lung_Opacity/masks/Lung_Opacity-2269.png,True,G04813,train,1ff94b806a09cd4c8a4ec50f11756cfb,d03d430b1d0a6f3f
Lung_Opacity-227.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-227.png,Lung_Opacity/masks/Lung_Opacity-227.png,True,G04814,train,b5a889541c3386fa105e093177f840e3,d42d4b5b4f1e2c62
Lung_Opacity-2270.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2270.png,Lung_Opacity/masks/Lung_Opacity-2270.png,True,G04815,train,ea7f2a7f54f316e54c510662e800f431,8075561e3d696f68
Lung_Opacity-2271.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2271.png,Lung_Opacity/masks/Lung_Opacity-2271.png,True,G04816,val,5687acfa71499bd4f9a6694e2ab9ee0c,c17916860f9cbd8d
Lung_Opacity-2272.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2272.png,Lung_Opacity/masks/Lung_Opacity-2272.png,True,G04817,train,124e232ce5b8a5950f004f599e33d11c,995d861a195e5a5e
Lung_Opacity-2273.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2273.png,Lung_Opacity/masks/Lung_Opacity-2273.png,True,G04818,train,3258116ac77ba4df4cf65ee301e6cf2d,992ae6423f5c694d
Lung_Opacity-2274.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2274.png,Lung_Opacity/masks/Lung_Opacity-2274.png,True,G04819,val,7bcf30f9738cfa2047b25de59f4d8f99,d12bd6d41c0c4b5f
Lung_Opacity-2275.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2275.png,Lung_Opacity/masks/Lung_Opacity-2275.png,True,G04820,train,52c4f00bceb404378626385cb83e6401,cd2d52430b1f4b3d
Lung_Opacity-2276.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2276.png,Lung_Opacity/masks/Lung_Opacity-2276.png,True,G04821,test,4420d27b8795197d8c06087d0ee8842e,d42d150b0f4e3c7e
Lung_Opacity-2277.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2277.png,Lung_Opacity/masks/Lung_Opacity-2277.png,True,G04822,train,0beb62a2f56fd6a4275c47f1edec0f53,dc4bc30f292e271d
Lung_Opacity-2278.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2278.png,Lung_Opacity/masks/Lung_Opacity-2278.png,True,G04823,train,7c9a76cb0bfd6e3d72a6622c51ecb774,d01b1f18792b2d6e
Lung_Opacity-2279.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2279.png,Lung_Opacity/masks/Lung_Opacity-2279.png,True,G04824,train,32d217adfc9432b07e025918b4456d65,993fc7961d005f1c
Lung_Opacity-228.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-228.png,Lung_Opacity/masks/Lung_Opacity-228.png,True,G04825,val,32a6e2fa5ca16bf1d73050a0d3ede24a,c43d114b4f1f196d
Lung_Opacity-2280.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2280.png,Lung_Opacity/masks/Lung_Opacity-2280.png,True,G04826,test,e5bfd6325823bd9012b196aca162f40f,9d610e66e7a09cd9
Lung_Opacity-2281.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2281.png,Lung_Opacity/masks/Lung_Opacity-2281.png,True,G04827,test,b4ba64205d554d431efa44696155528a,d42f07180f4f0f6d
Lung_Opacity-2282.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2282.png,Lung_Opacity/masks/Lung_Opacity-2282.png,True,G04828,train,fc2e708f0ddf8417f75c0a94748f1995,dd3c50d3434f0f52
Lung_Opacity-2283.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2283.png,Lung_Opacity/masks/Lung_Opacity-2283.png,True,G04829,train,8291cfff039d17a686789559b2721c8b,c1783e8590cbcf3c
Lung_Opacity-2284.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2284.png,Lung_Opacity/masks/Lung_Opacity-2284.png,True,G04830,train,c124074de2044417a72e73bc99936878,8447832d2d3f6b3c
Lung_Opacity-2285.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2285.png,Lung_Opacity/masks/Lung_Opacity-2285.png,True,G04831,train,615097879261e9004ffa682f99d3cfa3,c43f514d0e2d2d3e
Lung_Opacity-2286.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2286.png,Lung_Opacity/masks/Lung_Opacity-2286.png,True,G04832,train,672243cf389035c638c34d52b5f9b6fe,911d14141b3b3f3f
Lung_Opacity-2287.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2287.png,Lung_Opacity/masks/Lung_Opacity-2287.png,True,G04833,train,9aff832eedd50a79c60171922b8761c3,dc2f031d4c6c5ae3
Lung_Opacity-2288.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2288.png,Lung_Opacity/masks/Lung_Opacity-2288.png,True,G04834,train,78f0cc95ba11292df0d83f0dab4cd290,d42d02550b0e6f7f
Lung_Opacity-2289.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2289.png,Lung_Opacity/masks/Lung_Opacity-2289.png,True,G04835,train,e53dcd508d396bd40e09efd0162ba18b,897897565a8d5b58
Lung_Opacity-229.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-229.png,Lung_Opacity/masks/Lung_Opacity-229.png,True,G04836,train,c4c87d05b9f1e94e06c8e6beed41c22e,d57886475f181d1d
Lung_Opacity-2290.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2290.png,Lung_Opacity/masks/Lung_Opacity-2290.png,True,G04837,train,2eff544ae7458df1e66543a3cc6229b4,c93d165a1d5a195d
Lung_Opacity-2291.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2291.png,Lung_Opacity/masks/Lung_Opacity-2291.png,True,G04838,train,4bbb6a7c51327144de98554fd1e54e2c,813f571a0d292d7d
Lung_Opacity-2292.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2292.png,Lung_Opacity/masks/Lung_Opacity-2292.png,True,G04839,train,80250342acc4c4b3eada5e0e92552570,dd295752574a515a
Lung_Opacity-2293.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2293.png,Lung_Opacity/masks/Lung_Opacity-2293.png,True,G04840,train,2d460e066415e632c88a105cdcfc8d1d,887e145a195e5e4f
Lung_Opacity-2294.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2294.png,Lung_Opacity/masks/Lung_Opacity-2294.png,True,G04841,train,0cdfd145abaf88e9a6251eb5e0bafb2a,c42e16595cfceaa1
Lung_Opacity-2295.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2295.png,Lung_Opacity/masks/Lung_Opacity-2295.png,True,G04842,val,2668b55b13e043ae499c6d35342c3158,887d105e195b1fce
Lung_Opacity-2296.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2296.png,Lung_Opacity/masks/Lung_Opacity-2296.png,True,G04843,test,70bbc9b65b0d8ab58f8cc3bf56264eb3,817f04041f5f1f1f
Lung_Opacity-2297.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2297.png,Lung_Opacity/masks/Lung_Opacity-2297.png,True,G04844,val,090dfab70359e4f27b18fe7847e38727,d52487c36ae4e8f8
Lung_Opacity-2298.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2298.png,Lung_Opacity/masks/Lung_Opacity-2298.png,True,G04845,test,1f04abda8377a04028594bd4fb9012bb,d42ac54d4e0f2f2e
Lung_Opacity-2299.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2299.png,Lung_Opacity/masks/Lung_Opacity-2299.png,True,G04846,test,e48f9e2115f19f22eb99b84577b1e892,903fc25f58485b7a
Lung_Opacity-23.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-23.png,Lung_Opacity/masks/Lung_Opacity-23.png,True,G04847,train,339700a3e525fdf28cab0e3793175ab0,9564861e1f3c7c2e
Lung_Opacity-230.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-230.png,Lung_Opacity/masks/Lung_Opacity-230.png,True,G04848,val,83209d67f40abdfb3645df0a6746c4da,887dd243465b5b4b
Lung_Opacity-2300.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2300.png,Lung_Opacity/masks/Lung_Opacity-2300.png,True,G04849,train,e70d2712147aa68cb8b2ac1598fb7468,882bc65a5869f9f8
Lung_Opacity-2301.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2301.png,Lung_Opacity/masks/Lung_Opacity-2301.png,True,G04850,train,b2b5ee6377dd8950fc318ab790681302,c93b964a1d187b69
Lung_Opacity-2302.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2302.png,Lung_Opacity/masks/Lung_Opacity-2302.png,True,G04851,train,8b436e703ec202f401ff80b77a061492,d12be7d486c3891b
Lung_Opacity-2303.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2303.png,Lung_Opacity/masks/Lung_Opacity-2303.png,True,G04852,train,bced421bdc3984bc5fe45423d16e24d5,85781e001b5f795f
Lung_Opacity-2304.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2304.png,Lung_Opacity/masks/Lung_Opacity-2304.png,True,G04853,train,990cc437f8acd71e557cfb6990e6d6f0,d97907861ed83999
Lung_Opacity-2305.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2305.png,Lung_Opacity/masks/Lung_Opacity-2305.png,True,G04854,train,45272c7cd2ef4af3f371788dc5073649,85681f5772f2b036
Lung_Opacity-2306.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2306.png,Lung_Opacity/masks/Lung_Opacity-2306.png,True,G04855,train,bd82c3632a921935b062f7fedf72af19,913b451f184d5d5b
Lung_Opacity-2307.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2307.png,Lung_Opacity/masks/Lung_Opacity-2307.png,True,G04856,train,4f26a972a6c02ca8885c0d89eeca4a38,d169065e0b5f095f
Lung_Opacity-2308.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2308.png,Lung_Opacity/masks/Lung_Opacity-2308.png,True,G04857,val,b5c9382d06043712434615bf8d0a0f5d,d83b430b1f187e2e
Lung_Opacity-2309.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2309.png,Lung_Opacity/masks/Lung_Opacity-2309.png,True,G04858,val,49864463b881804ba02686ab668a613c,d52d0e0e1e5c4c6f
Lung_Opacity-231.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-231.png,Lung_Opacity/masks/Lung_Opacity-231.png,True,G04859,val,d27ae3059cbec0cf427844beda4631f5,992e061e1d4d1e5f
Lung_Opacity-2310.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2310.png,Lung_Opacity/masks/Lung_Opacity-2310.png,True,G04860,train,8db106556fc903c5e9b18f8ffdd75313,c13bd6d2080a5b7f
Lung_Opacity-2311.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2311.png,Lung_Opacity/masks/Lung_Opacity-2311.png,True,G04861,train,7280ff637dbe73e7e0f4b709b38f12f2,9c3d47c7165d4945
Lung_Opacity-2312.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2312.png,Lung_Opacity/masks/Lung_Opacity-2312.png,True,G04862,train,52ec0368ef191fe07b12f0ab9e173a5a,817b161a5b591b79
Lung_Opacity-2313.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2313.png,Lung_Opacity/masks/Lung_Opacity-2313.png,True,G04863,test,34e294a69e14ae9ae66b3afb668ceaa9,d83dd2429742d71b
Lung_Opacity-2314.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2314.png,Lung_Opacity/masks/Lung_Opacity-2314.png,True,G04864,train,2abaf6d74827d554eb38edbac59126c1,817fc61279697919
Lung_Opacity-2315.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2315.png,Lung_Opacity/masks/Lung_Opacity-2315.png,True,G04865,train,1505d51fe2b22e9dcc3f6ced92835520,dd38d7825e1a062f
Lung_Opacity-2316.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2316.png,Lung_Opacity/masks/Lung_Opacity-2316.png,True,G04866,test,56bc8410a860b28995e10a70b5a0f932,de34420f0f452f4f
Lung_Opacity-2317.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2317.png,Lung_Opacity/masks/Lung_Opacity-2317.png,True,G04867,train,4f7abd7e3b0ed2b58043ff74aac176c3,8c7f025b095f1a4f
Lung_Opacity-2318.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2318.png,Lung_Opacity/masks/Lung_Opacity-2318.png,True,G04868,train,c25f55ff94bffa169788ac4bfdc83770,d52a524a7f0cff01
Lung_Opacity-2319.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2319.png,Lung_Opacity/masks/Lung_Opacity-2319.png,True,G04869,test,9048175e7c6351c142359ef9edf16d45,801a9e5f197e5a3a
Lung_Opacity-232.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-232.png,Lung_Opacity/masks/Lung_Opacity-232.png,True,G04870,train,84ec6e48519bc1334ec72391aa96dd04,c83d4a0a0d2dffb2
Lung_Opacity-2320.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2320.png,Lung_Opacity/masks/Lung_Opacity-2320.png,True,G04871,train,12803ae1752fdfe9ae034944c8ced4f2,c42c53c34f0f1e5b
Lung_Opacity-2321.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2321.png,Lung_Opacity/masks/Lung_Opacity-2321.png,True,G04872,train,96fba76abf42c93956203f7be89067d8,d12bd71239393939
Lung_Opacity-2322.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2322.png,Lung_Opacity/masks/Lung_Opacity-2322.png,True,G04873,train,e53ae096404525b1121910aba2aa18b5,8158181f5e5c7e7a
Lung_Opacity-2323.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2323.png,Lung_Opacity/masks/Lung_Opacity-2323.png,True,G04874,train,164e24c2ddb44b9e98e27148e6dbe9d1,8c6c175ac15e68ed
Lung_Opacity-2324.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2324.png,Lung_Opacity/masks/Lung_Opacity-2324.png,True,G04875,train,f5f1164bd5c7fa8153add31e4200d97c,d52ad2474f4a0e4f
Lung_Opacity-2325.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2325.png,Lung_Opacity/masks/Lung_Opacity-2325.png,True,G04876,train,9bc38f5d443f6ba7f9916007d8e0c64e,813d140f5d6d6ae3
Lung_Opacity-2326.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2326.png,Lung_Opacity/masks/Lung_Opacity-2326.png,True,G04877,train,1e57b5d8633c07f261904183c2599b23,d46e835f4d0e0c4f
Lung_Opacity-2327.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2327.png,Lung_Opacity/masks/Lung_Opacity-2327.png,True,G04878,train,15c98af66c78c59221841aef012900eb,9c2cc6175a437e5a
Lung_Opacity-2328.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2328.png,Lung_Opacity/masks/Lung_Opacity-2328.png,True,G04879,train,fa0b74c660b3a563b02bfc9276bc8ed7,cc2d4b4b4f2c3cb4
Lung_Opacity-2329.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2329.png,Lung_Opacity/masks/Lung_Opacity-2329.png,True,G04880,test,0f6fb0c4f5dc95be75c0dab778e3c101,907b421e1e5f1b5c
Lung_Opacity-233.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-233.png,Lung_Opacity/masks/Lung_Opacity-233.png,True,G04881,train,4d0718edeed0111eac18c85019cd8334,c43d171f0e0e3c7c
Lung_Opacity-2330.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2330.png,Lung_Opacity/masks/Lung_Opacity-2330.png,True,G04882,train,c980ce0f4253d82c67bd6d4f4efb6524,817847141a5f5f3b
Lung_Opacity-2331.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2331.png,Lung_Opacity/masks/Lung_Opacity-2331.png,True,G04883,val,16606324149275a7666a61d26d6d32ac,907b0b3e3832b6c7
Lung_Opacity-2332.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2332.png,Lung_Opacity/masks/Lung_Opacity-2332.png,True,G04884,train,3007dfb02589addca1d3c350cad8dbfe,c42d0d0a0d6cfefa
Lung_Opacity-2333.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2333.png,Lung_Opacity/masks/Lung_Opacity-2333.png,True,G04885,val,e14f4037b8a64a96823572a6c45105a1,cc3d471b0a5f1c6c
Lung_Opacity-2334.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2334.png,Lung_Opacity/masks/Lung_Opacity-2334.png,True,G04886,val,47f68bafc96c237584cf804d2d62213c,916e0e1a4d78cbea
Lung_Opacity-2335.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2335.png,Lung_Opacity/masks/Lung_Opacity-2335.png,True,G04887,test,be33878a10a335d5586c49b1bef7602c,c42f0b3af1333333
Lung_Opacity-2336.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2336.png,Lung_Opacity/masks/Lung_Opacity-2336.png,True,G04888,test,10cc962e314c55283ee579173d8810c8,da7d42435f0a6f0a
Lung_Opacity-2337.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2337.png,Lung_Opacity/masks/Lung_Opacity-2337.png,True,G04889,train,2b21186d2c58a33fb4edd22ec2e4ae60,953807161e1d5f5e
Lung_Opacity-2338.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2338.png,Lung_Opacity/masks/Lung_Opacity-2338.png,True,G04890,train,a726d0fd77a02906d45da0602f63f57e,c3355256da165ab9
Lung_Opacity-2339.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2339.png,Lung_Opacity/masks/Lung_Opacity-2339.png,True,G04891,train,867df51a5b9aba1040a39b6ec650cbad,c42f074b454e4ecf
Lung_Opacity-234.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-234.png,Lung_Opacity/masks/Lung_Opacity-234.png,True,G04892,train,9450573eb55329db92fba34294b58461,c82d471a1c6d7c79
Lung_Opacity-2340.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2340.png,Lung_Opacity/masks/Lung_Opacity-2340.png,True,G04893,train,d290892b407746abdd6859d1e9cde738,c928175a1d4e597d
Lung_Opacity-2341.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2341.png,Lung_Opacity/masks/Lung_Opacity-2341.png,True,G04894,train,dfead1c603edf143090adfe683d34bf1,890f464e5b6beaa8
Lung_Opacity-2342.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2342.png,Lung_Opacity/masks/Lung_Opacity-2342.png,True,G04895,train,b5477c145a2930244a41468b38ba23d3,dd4a0f792a7232b2
Lung_Opacity-2343.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2343.png,Lung_Opacity/masks/Lung_Opacity-2343.png,True,G04896,test,d15a24f36a3891d4d7429cd3045c390e,9978071a796969e9
Lung_Opacity-2344.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2344.png,Lung_Opacity/masks/Lung_Opacity-2344.png,True,G04897,train,11613dde3f33383ddec4b51b16a82afe,c07f5a584a5b7b0a
Lung_Opacity-2345.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2345.png,Lung_Opacity/masks/Lung_Opacity-2345.png,True,G04898,train,95e0b4a39a02654c17075d77f66490d0,c42f3c76b252ab45
Lung_Opacity-2346.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2346.png,Lung_Opacity/masks/Lung_Opacity-2346.png,True,G04899,train,ca52c35eb50713bd25459cc52e7f7cd1,de2dc30f4f4c0e0b
Lung_Opacity-2347.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2347.png,Lung_Opacity/masks/Lung_Opacity-2347.png,True,G04900,train,9db68155e1127149af4751b0347afbdd,81788c965f59b81f
Lung_Opacity-2348.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2348.png,Lung_Opacity/masks/Lung_Opacity-2348.png,True,G04901,train,e71bf37c019d722afbed00162edaafae,c82d5e1a0d2c3cbf
Lung_Opacity-2349.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2349.png,Lung_Opacity/masks/Lung_Opacity-2349.png,True,G04902,train,a90d2bdc2765a1e275e1e0852d94e5c1,a83f820e1e1b3dbc
Lung_Opacity-235.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-235.png,Lung_Opacity/masks/Lung_Opacity-235.png,True,G04903,val,93529c1fabb89732db3990d9d70b6fc1,c03d5b439c0aed6d
Lung_Opacity-2350.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2350.png,Lung_Opacity/masks/Lung_Opacity-2350.png,True,G04904,val,271b0e5d6ffe78686de6e0b3cea709f0,d56b421b586d3e32
Lung_Opacity-2351.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2351.png,Lung_Opacity/masks/Lung_Opacity-2351.png,True,G04905,val,c916e0868f72f8668e67f0358a5c19c7,cc2f424e2f3b29a3
Lung_Opacity-2352.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2352.png,Lung_Opacity/masks/Lung_Opacity-2352.png,True,G04906,train,4da7886859170164266cc2124d2588f1,806e475c6c6e6aea
Lung_Opacity-2353.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2353.png,Lung_Opacity/masks/Lung_Opacity-2353.png,True,G04907,test,bd097859c1da1414e2a12be00804b0df,95780e855ab9e8c7
Lung_Opacity-2354.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2354.png,Lung_Opacity/masks/Lung_Opacity-2354.png,True,G04908,train,00a68bdd5467f0f58590732ffc2130a2,d52d424b4b4f0f4e
Lung_Opacity-2355.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2355.png,Lung_Opacity/masks/Lung_Opacity-2355.png,True,G04909,test,6390017a503925f882804a26ca4e2712,817a4e18f9e9b097
Lung_Opacity-2356.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2356.png,Lung_Opacity/masks/Lung_Opacity-2356.png,True,G04910,train,f0a1df843ab4cde872f9782ff5647af1,807fc25569697978
Lung_Opacity-2357.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2357.png,Lung_Opacity/masks/Lung_Opacity-2357.png,True,G04911,train,45d7736e07e0c1509c62a9be697b7bab,dd2ad20f5e494d58
Lung_Opacity-2358.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2358.png,Lung_Opacity/masks/Lung_Opacity-2358.png,True,G04912,val,a3b3564948f9a03be4b58e5917a2500c,897e07586f6560e5
Lung_Opacity-2359.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2359.png,Lung_Opacity/masks/Lung_Opacity-2359.png,True,G04913,train,149a11013797c66c95d344bf331d4573,d43f5b4b4f0d2c28
Lung_Opacity-236.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-236.png,Lung_Opacity/masks/Lung_Opacity-236.png,True,G04914,train,174b23961d7fbe9fe8438bef74588ede,903f434f154b4f1d
Lung_Opacity-2360.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2360.png,Lung_Opacity/masks/Lung_Opacity-2360.png,True,G04915,train,90cefce38587dd9070e3874182b00f4a,c83d52470f1d561f
Lung_Opacity-2361.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2361.png,Lung_Opacity/masks/Lung_Opacity-2361.png,True,G04916,train,748de92428b1b8ce2c1bd2221ff40d4a,d52ad24b1e5a4b1e
Lung_Opacity-2362.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2362.png,Lung_Opacity/masks/Lung_Opacity-2362.png,True,G04917,train,e6efdbbf1d3ba720f85a16a0b8ceacc6,c42ed5d38964f616
Lung_Opacity-2363.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2363.png,Lung_Opacity/masks/Lung_Opacity-2363.png,True,G04918,train,7f5be9e04d84c2b53f3556c56f43d056,913b861e3d1b3b19
Lung_Opacity-2364.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2364.png,Lung_Opacity/masks/Lung_Opacity-2364.png,True,G04919,train,67f5d8cfadd371164fb9a382b58b75db,c46f830d2c26fcf4
Lung_Opacity-2365.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2365.png,Lung_Opacity/masks/Lung_Opacity-2365.png,True,G04920,train,5c8739f82533dc1cbeae452796802002,913f070e1e0f0f1f
Lung_Opacity-2366.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2366.png,Lung_Opacity/masks/Lung_Opacity-2366.png,True,G04921,train,f5d999e7ea46f535576a4dc38fcc0883,d97907961e197a98
Lung_Opacity-2367.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2367.png,Lung_Opacity/masks/Lung_Opacity-2367.png,True,G04922,train,42a7a611599254c326237bd83fca2008,951a5e1a3a79287d
Lung_Opacity-2368.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2368.png,Lung_Opacity/masks/Lung_Opacity-2368.png,True,G04923,train,64aed7d0bbbc711f7f014c46aab38422,d428d94b3c3cb89b
Lung_Opacity-2369.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2369.png,Lung_Opacity/masks/Lung_Opacity-2369.png,True,G04924,val,352e1b5088e98f6b9631b03da4d706ce,dc38534b1f3a34aa
Lung_Opacity-237.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-237.png,Lung_Opacity/masks/Lung_Opacity-237.png,True,G04925,train,779809b2a7138d420516413272834c37,d03f0b161e5d4f1a
Lung_Opacity-2370.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2370.png,Lung_Opacity/masks/Lung_Opacity-2370.png,True,G04926,val,e12efa51057e8244d83e543d6575944b,9c2f030b4e2ceafa
Lung_Opacity-2371.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2371.png,Lung_Opacity/masks/Lung_Opacity-2371.png,True,G04927,test,5dac91ebfb07a0b15b1c19a8cea17ca8,887f8296f8e1e0f2
Lung_Opacity-2372.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2372.png,Lung_Opacity/masks/Lung_Opacity-2372.png,True,G04928,test,31b710aee7002f959ab0aef7cff851b9,997846161e4b5f39
Lung_Opacity-2373.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2373.png,Lung_Opacity/masks/Lung_Opacity-2373.png,True,G04929,test,ab5f9e6b437e8e53af3769227bafeb72,8928865a6b7d78cd
Lung_Opacity-2374.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2374.png,Lung_Opacity/masks/Lung_Opacity-2374.png,True,G04930,train,309de728c7644d612253488e85ca2274,d02d0f1d1c1c7cfa
Lung_Opacity-2375.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2375.png,Lung_Opacity/masks/Lung_Opacity-2375.png,True,G04931,train,417d9c77e141ea84866122a48c11b3d9,987f471a1a191f1e
Lung_Opacity-2376.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2376.png,Lung_Opacity/masks/Lung_Opacity-2376.png,True,G04932,train,f001d62f8858d9ec71d8ee5fb3407645,9135c790184f7b79
Lung_Opacity-2377.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2377.png,Lung_Opacity/masks/Lung_Opacity-2377.png,True,G04933,train,4e0ca1a4c73c179a8dd77c3bd36064fa,817f871a596d3c4a
Lung_Opacity-2378.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2378.png,Lung_Opacity/masks/Lung_Opacity-2378.png,True,G04934,train,62738938e61f8444fbd217f17705738a,9d0f031b0d6d3673
Lung_Opacity-2379.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2379.png,Lung_Opacity/masks/Lung_Opacity-2379.png,True,G04935,train,9fb2a9a9611307d47a7371c2f6227c9e,d62f41c8886ced4f
Lung_Opacity-238.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-238.png,Lung_Opacity/masks/Lung_Opacity-238.png,True,G04936,train,0225571d2f857c88355963ddf00a22eb,996f841e58696ce6
Lung_Opacity-2380.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2380.png,Lung_Opacity/masks/Lung_Opacity-2380.png,True,G04937,train,144c69ec4e57001dcf252b47895f7235,897c871a58457d79
Lung_Opacity-2381.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2381.png,Lung_Opacity/masks/Lung_Opacity-2381.png,True,G04938,train,4929262332ca90e0429c636e0ab2995c,8179e7c686969e58
Lung_Opacity-2382.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2382.png,Lung_Opacity/masks/Lung_Opacity-2382.png,True,G04939,train,c748247ce25ae776f6c18b058eb5e1a6,d02b57d35a0a4b3d
Lung_Opacity-2383.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2383.png,Lung_Opacity/masks/Lung_Opacity-2383.png,True,G04940,train,e5bf1ee8148d55f09e5cfccabe714a6f,917a411f1d193b3b
Lung_Opacity-2384.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2384.png,Lung_Opacity/masks/Lung_Opacity-2384.png,True,G04941,train,fc325564536529dbfdb39b6819522587,dd2a12190d5c7c7d
Lung_Opacity-2385.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2385.png,Lung_Opacity/masks/Lung_Opacity-2385.png,True,G04942,test,9b39d19496e9ed9d23219bdba67a38b1,95784f42190fecbc
Lung_Opacity-2386.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2386.png,Lung_Opacity/masks/Lung_Opacity-2386.png,True,G04943,val,2e3e2013a8b3361135b09078553b82da,817887161b5e5f5c
Lung_Opacity-2387.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2387.png,Lung_Opacity/masks/Lung_Opacity-2387.png,True,G04944,train,74d965942bce672fb2487e83d2de988a,c13e16435a1c7cf9
Lung_Opacity-2388.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2388.png,Lung_Opacity/masks/Lung_Opacity-2388.png,True,G04945,val,58e9e9f6af0d9c7cf3547b8268d49e4d,d03d3ad22dc56dc4
Lung_Opacity-2389.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2389.png,Lung_Opacity/masks/Lung_Opacity-2389.png,True,G04946,val,20967bfa8a6ec701c5dbfc29465edd02,cd2a510f6c2c2f9d
Lung_Opacity-239.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-239.png,Lung_Opacity/masks/Lung_Opacity-239.png,True,G04947,test,3389a63f2ea87e48e2d056cb491d3815,807c87c78a1b1f4f
Lung_Opacity-2390.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2390.png,Lung_Opacity/masks/Lung_Opacity-2390.png,True,G04948,test,a653fbff626a849f6e6985085ae819d1,9d2f4017174b4f5c
Lung_Opacity-2391.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2391.png,Lung_Opacity/masks/Lung_Opacity-2391.png,True,G04949,train,7e3dacdd9097d98ac521fc012d2a84ff,817f161a195d695d
Lung_Opacity-2392.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2392.png,Lung_Opacity/masks/Lung_Opacity-2392.png,True,G04950,train,554e56549bf8c325da2295269531d68c,93316e9a798664a7
Lung_Opacity-2393.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2393.png,Lung_Opacity/masks/Lung_Opacity-2393.png,True,G04951,train,bf6b60cf4fa9f15681579e4e9ca5f4df,c16907561f4c5b1f
Lung_Opacity-2394.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2394.png,Lung_Opacity/masks/Lung_Opacity-2394.png,True,G04952,train,e067f1e850c49ade82b0c0a7d4cea7e4,806f870b4d6d4d4f
Lung_Opacity-2395.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2395.png,Lung_Opacity/masks/Lung_Opacity-2395.png,True,G04953,train,1212f919687c0bb5f0497d5964432128,dc2f450b2c583ef1
Lung_Opacity-2396.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2396.png,Lung_Opacity/masks/Lung_Opacity-2396.png,True,G04954,val,349ae91b4cb2e4871e1534abdba9c19b,997946161e491f1f
Lung_Opacity-2397.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2397.png,Lung_Opacity/masks/Lung_Opacity-2397.png,True,G04955,test,934f85aee9e3aa95ec8bbc2a50dababc,816956561a1f5e3b
Lung_Opacity-2398.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2398.png,Lung_Opacity/masks/Lung_Opacity-2398.png,True,G04956,train,81c0d8d086111973e13e02646c9ce599,817f00174b175d5f
Lung_Opacity-2399.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2399.png,Lung_Opacity/masks/Lung_Opacity-2399.png,True,G04957,train,0ed4eeea0e67ed1c4b2375f8d9d091a8,c23d342d3f363233
Lung_Opacity-24.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-24.png,Lung_Opacity/masks/Lung_Opacity-24.png,True,G04958,test,0dfb3fab1973228925ba924a9df85717,857e47431b0f4f0b
Lung_Opacity-240.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-240.png,Lung_Opacity/masks/Lung_Opacity-240.png,True,G04959,val,ae0ace89e5dc811b184cbd0f0a939f2a,99398606175f0f1f
Lung_Opacity-2400.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2400.png,Lung_Opacity/masks/Lung_Opacity-2400.png,True,G04960,test,1326bc06129725d6147cafad1bfe553f,856c077b6133a397
Lung_Opacity-2401.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2401.png,Lung_Opacity/masks/Lung_Opacity-2401.png,True,G04961,train,dbc61a99b66fa4fcd1126cbe9a0468e8,917b841e1a7b3971
Lung_Opacity-2402.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2402.png,Lung_Opacity/masks/Lung_Opacity-2402.png,True,G04962,train,0491f09ad63c8f926aeac51966d061fe,c03d42534f1c78bf
Lung_Opacity-2403.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2403.png,Lung_Opacity/masks/Lung_Opacity-2403.png,True,G04963,train,3bcbf83f084dcd585a2144cf26ed9478,916a40191b1f5f5f
Lung_Opacity-2404.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2404.png,Lung_Opacity/masks/Lung_Opacity-2404.png,True,G04964,train,3def200d6d4762ff03557684b7871560,95e86886935e1f53
Lung_Opacity-2405.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2405.png,Lung_Opacity/masks/Lung_Opacity-2405.png,True,G04965,train,ed37bd286469b79cc2ab272bdad7c60c,957d6e823a8f6a90
Lung_Opacity-2406.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2406.png,Lung_Opacity/masks/Lung_Opacity-2406.png,True,G04966,train,745aada15e5a8c7c4ba9f42df258a4ce,856cc64b6d2c4bea
Lung_Opacity-2407.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2407.png,Lung_Opacity/masks/Lung_Opacity-2407.png,True,G04967,train,3a80c43af9991cef18be0dce6b668cc1,8d3f061f5d687c60
Lung_Opacity-2408.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2408.png,Lung_Opacity/masks/Lung_Opacity-2408.png,True,G04968,val,30716540812f1bdc6aaa15151738219c,883f16161a5b5d5e
Lung_Opacity-2409.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2409.png,Lung_Opacity/masks/Lung_Opacity-2409.png,True,G04969,train,2add32b6bd0145feeea8d99ef3bae97a,c17816de1a1b1f1b
Lung_Opacity-241.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-241.png,Lung_Opacity/masks/Lung_Opacity-241.png,True,G04970,train,8c8fbe31dda97b3248cea607b9513114,d42b474b0a2d6f2d
Lung_Opacity-2410.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2410.png,Lung_Opacity/masks/Lung_Opacity-2410.png,True,G04971,train,199ed62c080e7262e72cd3acfcc7cb93,d428f7430f1d5d49
Lung_Opacity-2411.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2411.png,Lung_Opacity/masks/Lung_Opacity-2411.png,True,G04972,train,901a1b9461944a746b6e1ca4033fc247,9969429e4bcd1bc9
Lung_Opacity-2412.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2412.png,Lung_Opacity/masks/Lung_Opacity-2412.png,True,G04973,train,ba0dab1e7844b43fd1a2b790d3c6e83d,987f421f5f4a1c4a
Lung_Opacity-2413.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2413.png,Lung_Opacity/masks/Lung_Opacity-2413.png,True,G04974,train,38545160d3d6bc0559bca9a8221e4a19,c93f16430b1c7b1d
Lung_Opacity-2414.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2414.png,Lung_Opacity/masks/Lung_Opacity-2414.png,True,G04975,train,5fc0288664448259a24671503fffcba3,987f045e1c1f194f
Lung_Opacity-2415.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2415.png,Lung_Opacity/masks/Lung_Opacity-2415.png,True,G04976,train,07f1e72a34fe56bc6dd362f3bd4e10f4,9929c61e1e4d3b39
Lung_Opacity-2416.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2416.png,Lung_Opacity/masks/Lung_Opacity-2416.png,True,G04977,train,c58fd729b040610b53fe707192ab115b,912957561f580b3f
Lung_Opacity-2417.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2417.png,Lung_Opacity/masks/Lung_Opacity-2417.png,True,G04978,train,d0594a567900b6aa964748c6a99d1c60,c13b4e4e5c1a4b4f
Lung_Opacity-2418.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2418.png,Lung_Opacity/masks/Lung_Opacity-2418.png,True,G04979,train,47eaf7a09124b601ae15ba1bf793766b,d96f464c48184f3f
Lung_Opacity-2419.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2419.png,Lung_Opacity/masks/Lung_Opacity-2419.png,True,G04980,train,c12b69c314ad18b6c8fb7b2865830915,d12bd749081b1f1f
Lung_Opacity-242.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-242.png,Lung_Opacity/masks/Lung_Opacity-242.png,True,G04981,train,e26077d265fb5faecbee0bd06abed0f1,dd3b121a5f185969
Lung_Opacity-2420.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2420.png,Lung_Opacity/masks/Lung_Opacity-2420.png,True,G04982,val,a04113559bd35cec6f9c78c860c1fc6e,d16942161f4d1f1f
Lung_Opacity-2421.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2421.png,Lung_Opacity/masks/Lung_Opacity-2421.png,True,G04983,train,12e59adbf81f9e0e62c2a670d5d2fd05,916e0c5f596c74c6
Lung_Opacity-2422.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2422.png,Lung_Opacity/masks/Lung_Opacity-2422.png,True,G04984,val,3bece55be736d98fe338812c1303d6cd,de600ccf6c6163e3
Lung_Opacity-2423.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2423.png,Lung_Opacity/masks/Lung_Opacity-2423.png,True,G04985,val,a20e8b0a95fbefb2015082017c1f6a86,80bd6d4a3e3dc235
Lung_Opacity-2424.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2424.png,Lung_Opacity/masks/Lung_Opacity-2424.png,True,G04986,test,5137df3d341f30d04fd23a27ee2ca598,d139025d5b0b5f1d
Lung_Opacity-2425.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2425.png,Lung_Opacity/masks/Lung_Opacity-2425.png,True,G04987,test,aacf1d1e2f065ff2788e4b1bcc391cb8,8d6c0a5ad9fae8e0
Lung_Opacity-2426.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2426.png,Lung_Opacity/masks/Lung_Opacity-2426.png,True,G04988,test,54bb58684e0d18026393b950b18ffe8c,807f06165e5f5959
Lung_Opacity-2427.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2427.png,Lung_Opacity/masks/Lung_Opacity-2427.png,True,G04989,train,b9d05c93616a9e4ad8bd84139b3f6bcb,897dc6125b79093d
Lung_Opacity-2428.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2428.png,Lung_Opacity/masks/Lung_Opacity-2428.png,True,G04990,train,ca2ee4096e64130c45a8d33646ed962e,c42f414b0f0f2e6f
Lung_Opacity-2429.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2429.png,Lung_Opacity/masks/Lung_Opacity-2429.png,True,G04991,val,41ac91f8c4d61ccca9d10b0416ea4683,d03d555f0a0a2f4f
Lung_Opacity-243.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-243.png,Lung_Opacity/masks/Lung_Opacity-243.png,True,G04992,train,bd78ff8e7fab0690fc428f93d63f9c8f,c13b16151a3a3b3f
Lung_Opacity-2430.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2430.png,Lung_Opacity/masks/Lung_Opacity-2430.png,True,G04993,train,7cf2f642abb435de1523fb61ad6c54ad,d02f07161f59786d
Lung_Opacity-2431.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2431.png,Lung_Opacity/masks/Lung_Opacity-2431.png,True,G04994,train,70753fa8de3ddb48839d71f9139bf11f,d5297a13cb4b2d0d
Lung_Opacity-2432.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2432.png,Lung_Opacity/masks/Lung_Opacity-2432.png,True,G04995,train,fc2685f81b489a14bd8957c9a1b9665f,d52b40571f494d5d
Lung_Opacity-2433.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2433.png,Lung_Opacity/masks/Lung_Opacity-2433.png,True,G04996,train,41f240fab1bb808f1ffa3608f06e34d4,d42d450a0b4f5d7d
Lung_Opacity-2434.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2434.png,Lung_Opacity/masks/Lung_Opacity-2434.png,True,G04997,train,3619900f067e458ebe336322ad4ad859,cd0d161b4b0d5f4d
Lung_Opacity-2435.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2435.png,Lung_Opacity/masks/Lung_Opacity-2435.png,True,G04998,train,9f40b693ffa5f1f76f5418f43dc3afe4,c03fc7131c6f0e2e
Lung_Opacity-2436.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2436.png,Lung_Opacity/masks/Lung_Opacity-2436.png,True,G04999,train,febf4345a30a74988760e99bc6da23bc,cd3e161b9f0a1f03
Lung_Opacity-2437.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2437.png,Lung_Opacity/masks/Lung_Opacity-2437.png,True,G05000,train,375a98f2729862203e46b7103277b4ff,d03b43c65e8b6b89
Lung_Opacity-2438.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2438.png,Lung_Opacity/masks/Lung_Opacity-2438.png,True,G05001,train,16dfbb111c9a7d59ee5740eb79c2c2c6,dd2f43535f4c484c
Lung_Opacity-2439.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2439.png,Lung_Opacity/masks/Lung_Opacity-2439.png,True,G05002,train,3b28fa2bc2701e3ff025537d76177fde,9d2ae24b4f4b4d0d
Lung_Opacity-244.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-244.png,Lung_Opacity/masks/Lung_Opacity-244.png,True,G05003,train,0b6138dfc0ba3c0064a1bd74eb26a832,d53b524f0c292d3e
Lung_Opacity-2440.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2440.png,Lung_Opacity/masks/Lung_Opacity-2440.png,True,G05004,val,f7a7d467d8986c0b4edd8fcf2f9cdc34,dd7c47531b090f58
Lung_Opacity-2441.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2441.png,Lung_Opacity/masks/Lung_Opacity-2441.png,True,G05005,val,19c90fb04543291d9ed69eecfe4ccecc,987a535b55525157
Lung_Opacity-2442.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2442.png,Lung_Opacity/masks/Lung_Opacity-2442.png,True,G05006,train,695c6853f5b096384636a2277c63e6c6,987dc21a594f3ac9
Lung_Opacity-2443.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2443.png,Lung_Opacity/masks/Lung_Opacity-2443.png,True,G05007,test,51889ae2105bbc2284fae04adfd9f46e,8179961a1a793a7b
Lung_Opacity-2444.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2444.png,Lung_Opacity/masks/Lung_Opacity-2444.png,True,G05008,test,207b6c5c5ea1bf708e8972e9f4dfafec,cc2cd34b4f682d0f
Lung_Opacity-2445.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2445.png,Lung_Opacity/masks/Lung_Opacity-2445.png,True,G05009,test,18637cfcd07fdc4eb379d95946640c91,9139c6861a7ded1a
Lung_Opacity-2446.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2446.png,Lung_Opacity/masks/Lung_Opacity-2446.png,True,G05010,val,48570c39f371a1f333f4489aeddfafca,836aac90d24b6f4f
Lung_Opacity-2447.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2447.png,Lung_Opacity/masks/Lung_Opacity-2447.png,True,G05011,train,3ad97239e56a62b909fae29ab12eebdf,807e175a7979e8e0
Lung_Opacity-2448.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2448.png,Lung_Opacity/masks/Lung_Opacity-2448.png,True,G05012,train,3d4aec03ca34dcf46011b9a02491d860,d538121d193bbaa7
Lung_Opacity-2449.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2449.png,Lung_Opacity/masks/Lung_Opacity-2449.png,True,G05013,train,18aebad6d3a11369d80f63a24ba9d62b,d03b5b1a5e1a1b1b
Lung_Opacity-245.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-245.png,Lung_Opacity/masks/Lung_Opacity-245.png,True,G05014,train,567610637919b02966178459c219b55f,944f071c4ccafaf2
Lung_Opacity-2450.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2450.png,Lung_Opacity/masks/Lung_Opacity-2450.png,True,G05015,train,bc05cc333940b7c66635cd35ec92330f,956868977a359762
Lung_Opacity-2451.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2451.png,Lung_Opacity/masks/Lung_Opacity-2451.png,True,G05016,train,b69a676c0a6c854c4065a90d636bd689,cc2cd24b555e5d0e
Lung_Opacity-2452.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2452.png,Lung_Opacity/masks/Lung_Opacity-2452.png,True,G05017,train,667cd7109d854b39cd3d468d3dc17d14,df7d4a3968785828
Lung_Opacity-2453.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2453.png,Lung_Opacity/masks/Lung_Opacity-2453.png,True,G05018,train,7d86663fc81d255485f19091364e1879,c235774b0b412f6e
Lung_Opacity-2454.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2454.png,Lung_Opacity/masks/Lung_Opacity-2454.png,True,G05019,train,bc8c6792f112b3ab1e7fc3e97cc1d978,c42f164f4e6c6c6c
Lung_Opacity-2455.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2455.png,Lung_Opacity/masks/Lung_Opacity-2455.png,True,G05020,train,101d6d5686f527776fdb580538262ab8,997d86161a0d5b79
Lung_Opacity-2456.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2456.png,Lung_Opacity/masks/Lung_Opacity-2456.png,True,G05021,train,b72b0f4223833f54e9b114f8110b3c4d,dc2c175a4d5c686e
Lung_Opacity-2457.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2457.png,Lung_Opacity/masks/Lung_Opacity-2457.png,True,G05022,train,edbdd7bbaa4699980cc646f0d67ff7d6,c85d8f4a092d7d3c
Lung_Opacity-2458.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2458.png,Lung_Opacity/masks/Lung_Opacity-2458.png,True,G05023,val,ca68952b174e542377081cfa2e0e714d,976168da69346736
Lung_Opacity-2459.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2459.png,Lung_Opacity/masks/Lung_Opacity-2459.png,True,G05024,train,249f0f8b17fc408e6234baf35e862569,91384756184dedf9
Lung_Opacity-246.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-246.png,Lung_Opacity/masks/Lung_Opacity-246.png,True,G05025,val,96098e9c0a0b6bfda5b25675d02000ab,9c7cc3531f6c6c0c
Lung_Opacity-2460.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2460.png,Lung_Opacity/masks/Lung_Opacity-2460.png,True,G05026,train,8fa850491ecc3863914fd643665702e7,c46de7d059786968
Lung_Opacity-2461.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2461.png,Lung_Opacity/masks/Lung_Opacity-2461.png,True,G05027,train,6d71b8c0acc256222c22a29c93862efa,d03dd3d55c003c7e
Lung_Opacity-2462.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2462.png,Lung_Opacity/masks/Lung_Opacity-2462.png,True,G05028,test,17c42f170ffdc1552711e31bab555832,897b1612597959cb
Lung_Opacity-2463.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2463.png,Lung_Opacity/masks/Lung_Opacity-2463.png,True,G05029,train,c3852e76cea5237e953bd3dd29c0294f,907be55e144e5a39
Lung_Opacity-2464.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2464.png,Lung_Opacity/masks/Lung_Opacity-2464.png,True,G05030,train,60fc6b42499f7826fd918e3bd8359678,917a121f4b7aeca4
Lung_Opacity-2465.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2465.png,Lung_Opacity/masks/Lung_Opacity-2465.png,True,G05031,val,3dd8248ba30eb5ffa3f99f6fc7bc238b,996f065b411d1f47
Lung_Opacity-2466.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2466.png,Lung_Opacity/masks/Lung_Opacity-2466.png,True,G05032,test,07fe11d8240d376aca4d257f02423cee,903b431e3c6c6e67
Lung_Opacity-2467.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2467.png,Lung_Opacity/masks/Lung_Opacity-2467.png,True,G05033,test,ee2370b573ff205be754efcb1e235b96,c96d464b593c466b
Lung_Opacity-2468.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2468.png,Lung_Opacity/masks/Lung_Opacity-2468.png,True,G05034,train,112692841b0c030643a4c75233adf539,d43fc30f0a0f0f47
Lung_Opacity-2469.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2469.png,Lung_Opacity/masks/Lung_Opacity-2469.png,True,G05035,train,047979ddbd5a129bf599bacb451c7660,fd2d87120b2d3c6c
Lung_Opacity-247.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-247.png,Lung_Opacity/masks/Lung_Opacity-247.png,True,G05036,train,f4685dc2fc76ae2e7ca18057091811ce,dd2c42c31b514f4f
Lung_Opacity-2470.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2470.png,Lung_Opacity/masks/Lung_Opacity-2470.png,True,G05037,train,e3e231a170a0372f42ff0064e7ffb0ac,d16d425b494f595a
Lung_Opacity-2471.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2471.png,Lung_Opacity/masks/Lung_Opacity-2471.png,True,G05038,train,a3f42e6f2e438e68cb9feb45f908df08,917be4025e6d7929
Lung_Opacity-2472.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2472.png,Lung_Opacity/masks/Lung_Opacity-2472.png,True,G05039,train,051affb8e1235c6cd227565661e22002,d43d42033f0a6d7d
Lung_Opacity-2473.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2473.png,Lung_Opacity/masks/Lung_Opacity-2473.png,True,G05040,train,84964c7d16c0d8c2e09dd51a69d61e76,9c2b470e4f1f484f
Lung_Opacity-2474.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2474.png,Lung_Opacity/masks/Lung_Opacity-2474.png,True,G05041,train,af686cec1a14a79ee598ac9919d688da,903a474f1cda531f
Lung_Opacity-2475.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2475.png,Lung_Opacity/masks/Lung_Opacity-2475.png,True,G05042,train,ea2fd42911685002427788231c77b998,947eab578b050d2d
Lung_Opacity-2476.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2476.png,Lung_Opacity/masks/Lung_Opacity-2476.png,True,G05043,train,7e1c231a3f761f10145d024303f9bfc4,996c0c5f4deac6c2
Lung_Opacity-2477.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2477.png,Lung_Opacity/masks/Lung_Opacity-2477.png,True,G05044,val,a8a804d4bef51218d73783721c602979,883d061e595d595f
Lung_Opacity-2478.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2478.png,Lung_Opacity/masks/Lung_Opacity-2478.png,True,G05045,val,2608b82083ab02fc47dfa38e5f4903a5,9179c6161a17f85b
Lung_Opacity-2479.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2479.png,Lung_Opacity/masks/Lung_Opacity-2479.png,True,G05046,test,9c94e44b84d5c13113bbe0cfaba6bd61,917a905b1f193b93
Lung_Opacity-248.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-248.png,Lung_Opacity/masks/Lung_Opacity-248.png,True,G05047,test,832d4f28fb60e43d481d9a4d3fdb8416,846e474f6c6e6e60
Lung_Opacity-2480.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2480.png,Lung_Opacity/masks/Lung_Opacity-2480.png,True,G05048,test,b67e17edb465401912c028c97923fb37,c03c13d15bca3eda
Lung_Opacity-2481.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2481.png,Lung_Opacity/masks/Lung_Opacity-2481.png,True,G05049,train,c5983c98c576d7d3de0afed84a402442,8c2e070f5c5a5e5e
Lung_Opacity-2482.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2482.png,Lung_Opacity/masks/Lung_Opacity-2482.png,True,G05050,train,6f414197c76257514a59aaa8e07cf110,816817571b5e5e5a
Lung_Opacity-2483.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2483.png,Lung_Opacity/masks/Lung_Opacity-2483.png,True,G05051,train,d20bcc0bd244ae1e0a73c4b1261452cc,cc69930f1a17595e
Lung_Opacity-2484.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2484.png,Lung_Opacity/masks/Lung_Opacity-2484.png,True,G05052,train,6fa1817b0a3bd08d6d4f1c48e93d1cb0,817ae017fdcfc450
Lung_Opacity-2485.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2485.png,Lung_Opacity/masks/Lung_Opacity-2485.png,True,G05053,train,42da0064d8d97348b80c683d1eddd7fa,dc2dd38b5d285c1c
Lung_Opacity-2486.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2486.png,Lung_Opacity/masks/Lung_Opacity-2486.png,True,G05054,train,a6ee0f66332a81d5c06af11ca4a4274f,d9bdc716041e390f
Lung_Opacity-2487.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2487.png,Lung_Opacity/masks/Lung_Opacity-2487.png,True,G05055,train,9da575441295850f9dcc9bde8bbf5290,83285a5f5b1b5939
Lung_Opacity-2488.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2488.png,Lung_Opacity/masks/Lung_Opacity-2488.png,True,G05056,train,b6bb7fa665e0fa30924e219fd1907591,817fc61e7a4b2730
Lung_Opacity-2489.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2489.png,Lung_Opacity/masks/Lung_Opacity-2489.png,True,G05057,train,0f5e6e74c92516f8007a673b9f10e3b5,ce242f3d70f1990d
Lung_Opacity-249.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-249.png,Lung_Opacity/masks/Lung_Opacity-249.png,True,G05058,train,01ef4914f59bb67a4b7a272cca48b055,807fae82171d2f1d
Lung_Opacity-2490.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2490.png,Lung_Opacity/masks/Lung_Opacity-2490.png,True,G05059,train,a298ab7345353c171edd7b0fb4205df6,9a684c5e5c6969ec
Lung_Opacity-2491.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2491.png,Lung_Opacity/masks/Lung_Opacity-2491.png,True,G05060,train,dfcd70d03f5c742cf491b0e1cf43cad2,d03b07171e185b7b
Lung_Opacity-2492.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2492.png,Lung_Opacity/masks/Lung_Opacity-2492.png,True,G05061,train,69a57c8968d145c0b87b06a3eb2c4214,8d0f074b4b68e9e9
Lung_Opacity-2493.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2493.png,Lung_Opacity/masks/Lung_Opacity-2493.png,True,G05062,train,50482a37315df636e73f4a78acc1f54e,916cc75a1a4e2b3b
Lung_Opacity-2494.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2494.png,Lung_Opacity/masks/Lung_Opacity-2494.png,True,G05063,val,91ed6276c9793c1ae3d383542feef1b2,d42fc1c30d4f155d
Lung_Opacity-2495.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2495.png,Lung_Opacity/masks/Lung_Opacity-2495.png,True,G05064,val,a6254bc8489077520a672afdf38b8a11,817a18181b7f5b5b
Lung_Opacity-2496.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2496.png,Lung_Opacity/masks/Lung_Opacity-2496.png,True,G05065,val,4345e43fb42c782421b7f06dcac609bd,816a829adff8b2b2
Lung_Opacity-2497.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2497.png,Lung_Opacity/masks/Lung_Opacity-2497.png,True,G05066,test,1f5d77cf28e89de9644178dcad132fc8,9179405f79295b1e
Lung_Opacity-2498.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2498.png,Lung_Opacity/masks/Lung_Opacity-2498.png,True,G05067,test,6bd8dc4b97d9b4aa00bbf9a3d23d41d9,d56e054c4b4e6e4e
Lung_Opacity-2499.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2499.png,Lung_Opacity/masks/Lung_Opacity-2499.png,True,G05068,test,a54c55355d87676fb0c74417cda3672c,d42e5b490d2c3ead
Lung_Opacity-25.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-25.png,Lung_Opacity/masks/Lung_Opacity-25.png,True,G05069,train,5c39f76d7750e55e84cc89d968e51e23,9d7c86161d1f1d16
Lung_Opacity-250.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-250.png,Lung_Opacity/masks/Lung_Opacity-250.png,True,G05070,val,1939f36e718317586c3465faad11ffed,c03e3d4d4e4e2e2e
Lung_Opacity-2500.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2500.png,Lung_Opacity/masks/Lung_Opacity-2500.png,True,G05071,train,8d06ea553d18fdb2d3e2c9241371dda6,cc0b030f4f4e4f4f
Lung_Opacity-2501.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2501.png,Lung_Opacity/masks/Lung_Opacity-2501.png,True,G05072,train,eb096cbe7e545531aa30af15790f7713,cc27834f4c4a65eb
Lung_Opacity-2502.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2502.png,Lung_Opacity/masks/Lung_Opacity-2502.png,True,G05073,train,4bde91e435a554d172ea8b16015be3e7,d22f054e0e6e2e6e
Lung_Opacity-2503.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2503.png,Lung_Opacity/masks/Lung_Opacity-2503.png,True,G05074,train,e64e916c5dce7c4786404d6871a70c77,c43c3bc34b0a6e6d
Lung_Opacity-2504.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2504.png,Lung_Opacity/masks/Lung_Opacity-2504.png,True,G05075,train,ae7cca637778d91eefdbdf45cfbf8436,c54f168a6d6cf470
Lung_Opacity-2505.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2505.png,Lung_Opacity/masks/Lung_Opacity-2505.png,True,G05076,train,aa2d96a1a4396cb01591defa97bf6fee,946e454e0c5f0a7f
Lung_Opacity-2506.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2506.png,Lung_Opacity/masks/Lung_Opacity-2506.png,True,G05077,train,649b6e6ddc087301819e52250f26c987,d92dc2171f4c1b1d
Lung_Opacity-2507.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2507.png,Lung_Opacity/masks/Lung_Opacity-2507.png,True,G05078,train,532ba1029e859bdff7fd872cd2933e33,cc2f96434b0c5e6d
Lung_Opacity-2508.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2508.png,Lung_Opacity/masks/Lung_Opacity-2508.png,True,G05079,train,feb6693d2b38468d88b1c245b96a643f,cc0c42535f4d697b
Lung_Opacity-2509.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2509.png,Lung_Opacity/masks/Lung_Opacity-2509.png,True,G05080,train,fa02128f4066b4dd4d873836f2e752cc,c43e4f4c18e987a7
Lung_Opacity-251.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-251.png,Lung_Opacity/masks/Lung_Opacity-251.png,True,G05081,train,af4804e82a188a32dadde5d1e6671527,896e065a4f4c5e4f
Lung_Opacity-2510.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2510.png,Lung_Opacity/masks/Lung_Opacity-2510.png,True,G05082,train,89518503a4e3c82df09add31adc223f5,d0395b7b7a343434
Lung_Opacity-2511.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2511.png,Lung_Opacity/masks/Lung_Opacity-2511.png,True,G05083,train,c533063c63c17c6e6f05892432b24ee9,d42eb3531b1f921a
Lung_Opacity-2512.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2512.png,Lung_Opacity/masks/Lung_Opacity-2512.png,True,G05084,val,ac9d4563d2543c1f61da2b2bd00c3ee2,913f065e1a157f38
Lung_Opacity-2513.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2513.png,Lung_Opacity/masks/Lung_Opacity-2513.png,True,G05085,val,a67aeec11a83959596d64cbe353e0209,cc1c430b6c6e3e2f
Lung_Opacity-2514.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2514.png,Lung_Opacity/masks/Lung_Opacity-2514.png,True,G05086,test,42541af676eb2d0ce764afbf2a01ee14,cc7c52434b4b4f0f
Lung_Opacity-2515.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2515.png,Lung_Opacity/masks/Lung_Opacity-2515.png,True,G05087,train,826c14393c52079dae48fc17981b3eac,8c7c4c4e4d0e8ecf
Lung_Opacity-2516.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2516.png,Lung_Opacity/masks/Lung_Opacity-2516.png,True,G05088,train,422c851f6c621d69b0cbd89693cd22c7,d03d4b5c0f0a3db9
Lung_Opacity-2517.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2517.png,Lung_Opacity/masks/Lung_Opacity-2517.png,True,G05089,test,9360af467c13e925bfc168df1a7aa441,9b7b703a333a2a2a
Lung_Opacity-2518.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2518.png,Lung_Opacity/masks/Lung_Opacity-2518.png,True,G05090,test,f840a6a0a51d83748162289780e241e1,d57a2f5783418707
Lung_Opacity-2519.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2519.png,Lung_Opacity/masks/Lung_Opacity-2519.png,True,G05091,train,86ebf806a905e979521ebd6b33cc50fd,d52a015f6d7878f0
Lung_Opacity-252.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-252.png,Lung_Opacity/masks/Lung_Opacity-252.png,True,G05092,train,93af37fdf6fc3bb1cb3bf386f2ba3a16,816ac41e5c5f5b59
Lung_Opacity-2520.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2520.png,Lung_Opacity/masks/Lung_Opacity-2520.png,True,G05093,train,aa1489a731b1386e79f74b9606267245,dc4d021e4b3d6c7c
Lung_Opacity-2521.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2521.png,Lung_Opacity/masks/Lung_Opacity-2521.png,True,G05094,train,944cc6fce5beda6d35ad268420cfd3c0,d46f85438b0f0e2f
Lung_Opacity-2522.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2522.png,Lung_Opacity/masks/Lung_Opacity-2522.png,True,G05095,train,8ee124a5ac762a7f9f32fe1035244b2b,9c6e061f095d5c7a
Lung_Opacity-2523.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2523.png,Lung_Opacity/masks/Lung_Opacity-2523.png,True,G05096,train,d39dc2dd06173bab54e7f04545c2a23f,997843c31e455d3d
Lung_Opacity-2524.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2524.png,Lung_Opacity/masks/Lung_Opacity-2524.png,True,G05097,train,d4efe9ffe7f30a1e6045ddba3cd50446,99611a1e5f5af868
Lung_Opacity-2525.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2525.png,Lung_Opacity/masks/Lung_Opacity-2525.png,True,G05098,train,b32e72212d3ed8e6b4451c75be861f2b,8817161e5a5f1a7e
Lung_Opacity-2526.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2526.png,Lung_Opacity/masks/Lung_Opacity-2526.png,True,G05099,train,f0e6991a0bcf5556068705033283f69e,982e461f5e4f1d49
Lung_Opacity-2527.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2527.png,Lung_Opacity/masks/Lung_Opacity-2527.png,True,G05100,train,30c5832fbc7703f2a9988634bd0f5ed4,95680e1f5b7d6868
Lung_Opacity-2528.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2528.png,Lung_Opacity/masks/Lung_Opacity-2528.png,True,G05101,train,012c86d4bd0a3c78559759ebc3acf544,817b161a19797b39
Lung_Opacity-2529.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2529.png,Lung_Opacity/masks/Lung_Opacity-2529.png,True,G05102,train,90089f576411e6c4afa9f1b92e7ba53a,dc2ec3931f55281d
Lung_Opacity-253.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-253.png,Lung_Opacity/masks/Lung_Opacity-253.png,True,G05103,val,3a2da649a2fa8e9e49274650ef53ee42,917a571b0b390737
Lung_Opacity-2530.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2530.png,Lung_Opacity/masks/Lung_Opacity-2530.png,True,G05104,val,bbe4cb5e9373e8d5a33780439c2c03ad,cd2b95971e1c521b
Lung_Opacity-2531.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2531.png,Lung_Opacity/masks/Lung_Opacity-2531.png,True,G05105,test,fcf65de55989406d13162345ce61574f,8179101b79595f1f
Lung_Opacity-2532.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2532.png,Lung_Opacity/masks/Lung_Opacity-2532.png,True,G05106,val,02bc2edcb658ddff154e815cdada8ef0,d506034b2e6e6b7a
Lung_Opacity-2533.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2533.png,Lung_Opacity/masks/Lung_Opacity-2533.png,True,G05107,test,d26122448e9c9e1c0433b6b860d566f1,c93e964890c9ddc7
Lung_Opacity-2534.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2534.png,Lung_Opacity/masks/Lung_Opacity-2534.png,True,G05108,test,040e1fa528a3b320ee570f520dc128f0,c52a7e4e495d5a58
Lung_Opacity-2535.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2535.png,Lung_Opacity/masks/Lung_Opacity-2535.png,True,G05109,train,0383665ee6ed126d01215982cb6448b0,985fda18813aa73d
Lung_Opacity-2536.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2536.png,Lung_Opacity/masks/Lung_Opacity-2536.png,True,G05110,train,beb2d88972674d5990b85b94de2edab7,983dc7061e91cd8f
Lung_Opacity-2537.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2537.png,Lung_Opacity/masks/Lung_Opacity-2537.png,True,G05111,train,e27344d6f647b9d5c55a0774c33fac21,8ab9e0565ad45c5b
Lung_Opacity-2538.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2538.png,Lung_Opacity/masks/Lung_Opacity-2538.png,True,G05112,train,dbc75335a173530bdee9552182eccdd4,854e0f5bcb6a61e1
Lung_Opacity-2539.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2539.png,Lung_Opacity/masks/Lung_Opacity-2539.png,True,G05113,train,8ee75756165e727797cc4f7e50d0a7a3,807a565e3c6f2527
Lung_Opacity-254.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-254.png,Lung_Opacity/masks/Lung_Opacity-254.png,True,G05114,train,ab4d067dd1efab9b064a297deae3e022,ceb9fb05a03d90b4
Lung_Opacity-2540.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2540.png,Lung_Opacity/masks/Lung_Opacity-2540.png,True,G05115,train,bff7f832d8bbe351fff4c52a05070eb6,913a445b7b0b7a39
Lung_Opacity-2541.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2541.png,Lung_Opacity/masks/Lung_Opacity-2541.png,True,G05116,train,346f06d72bfb3a2aac6e037eaf0251f3,917a461e1a5b7a3a
Lung_Opacity-2542.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2542.png,Lung_Opacity/masks/Lung_Opacity-2542.png,True,G05117,train,c868c197e55506b315722b046cb917a2,c03d53134b0f6f2d
Lung_Opacity-2543.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2543.png,Lung_Opacity/masks/Lung_Opacity-2543.png,True,G05118,train,16949f433dd5996adfb5ae3231a1b178,801f105f5a5e5e5e
Lung_Opacity-2544.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2544.png,Lung_Opacity/masks/Lung_Opacity-2544.png,True,G05119,train,890f19cbe1a2e5bc43020e61f993d90c,897fa4d6c259a1d2
Lung_Opacity-2545.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2545.png,Lung_Opacity/masks/Lung_Opacity-2545.png,True,G05120,train,ff7adf0f333ead697d2b9aa58b35a2e4,996d061a4d7c4c7d
Lung_Opacity-2546.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2546.png,Lung_Opacity/masks/Lung_Opacity-2546.png,True,G05121,train,c0b152c3bc93fb8fd0359b6f4a4b8308,95697e9a61866696
Lung_Opacity-2547.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2547.png,Lung_Opacity/masks/Lung_Opacity-2547.png,True,G05122,train,fab0463bdef85f3dc08b92a61ecb0291,d528871f1b0f0f1d
Lung_Opacity-2548.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2548.png,Lung_Opacity/masks/Lung_Opacity-2548.png,True,G05123,val,b1e27e9870f602edde2c204612c82f83,d93952921e1e3a3b
Lung_Opacity-2549.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2549.png,Lung_Opacity/masks/Lung_Opacity-2549.png,True,G05124,val,b94dff4b974e63f9647b15bf6d5ecb9d,9f6004131e7e7a7c
Lung_Opacity-255.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-255.png,Lung_Opacity/masks/Lung_Opacity-255.png,True,G05125,val,3255b6ebe9b305f200470420859897ea,c03d531355595e4f
Lung_Opacity-2550.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2550.png,Lung_Opacity/masks/Lung_Opacity-2550.png,True,G05126,test,623efa44105c4f9bfbcde385a6639642,817c971a191d7c5e
Lung_Opacity-2551.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2551.png,Lung_Opacity/masks/Lung_Opacity-2551.png,True,G05127,test,97cac1124d9895d1e05180e89af39b40,d03f070f1d1d3c3c
Lung_Opacity-2552.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2552.png,Lung_Opacity/masks/Lung_Opacity-2552.png,True,G05128,test,f66da7aba7bbcc57a8069317b30934d8,9918165f587978ea
Lung_Opacity-2553.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2553.png,Lung_Opacity/masks/Lung_Opacity-2553.png,True,G05129,train,bb04c8cda3daf8a20e364389858f978a,896c065a4d7d68ee
Lung_Opacity-2554.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2554.png,Lung_Opacity/masks/Lung_Opacity-2554.png,True,G05130,train,d1d66f1ebbe20f3a5487c5db82605786,c1391e944b1b3f5a
Lung_Opacity-2555.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2555.png,Lung_Opacity/masks/Lung_Opacity-2555.png,True,G05131,train,8458d43e60f8cfaa1c5ed3273895d3a1,e86dc71ad3187945
Lung_Opacity-2556.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2556.png,Lung_Opacity/masks/Lung_Opacity-2556.png,True,G05132,train,294baf06a024e4b170b8e0ee1b4d1d4e,c03d5b195a1c3a3f
Lung_Opacity-2557.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2557.png,Lung_Opacity/masks/Lung_Opacity-2557.png,True,G05133,train,7d012859163080852dba43c402405287,dc29521b4d0d3e3d
Lung_Opacity-2558.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2558.png,Lung_Opacity/masks/Lung_Opacity-2558.png,True,G05134,train,1869e9695d5233a60d5c141c4c7e23bb,897d061c584f3d5b
Lung_Opacity-2559.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2559.png,Lung_Opacity/masks/Lung_Opacity-2559.png,True,G05135,train,521fc16f653a885b1f6864e9ad6defbf,95208a791b3b3b1f
Lung_Opacity-256.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-256.png,Lung_Opacity/masks/Lung_Opacity-256.png,True,G05136,train,8cf714f0022ca06b728d9f0184feaee1,c57c0f0c37313373
Lung_Opacity-2560.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2560.png,Lung_Opacity/masks/Lung_Opacity-2560.png,True,G05137,train,d691b9ff4f5f9f3f5e76727a08e2568a,d528d71f4e5d4c0a
Lung_Opacity-2561.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2561.png,Lung_Opacity/masks/Lung_Opacity-2561.png,True,G05138,train,1c4de6b87d5c68fb651b04e8d2bc6b50,903b1f121a595b5f
Lung_Opacity-2562.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2562.png,Lung_Opacity/masks/Lung_Opacity-2562.png,True,G05139,val,84914a1b290a719a6198de32ad98f8b3,d42f0149c70f4f4f
Lung_Opacity-2563.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2563.png,Lung_Opacity/masks/Lung_Opacity-2563.png,True,G05140,train,95f2bb23226c7cb808456c920972cd48,8d6a165a1e4b1b3e
Lung_Opacity-2564.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2564.png,Lung_Opacity/masks/Lung_Opacity-2564.png,True,G05141,train,087af5e5cd5dd622e0ef14b2047e9f8e,9a491c5e7942c757
Lung_Opacity-2565.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2565.png,Lung_Opacity/masks/Lung_Opacity-2565.png,True,G05142,train,d0323b9c8d7a1f882afb646f30814327,c17816111d4f7b3d
Lung_Opacity-2566.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2566.png,Lung_Opacity/masks/Lung_Opacity-2566.png,True,G05143,train,ae5735414de150c52ba07b91860e8027,cd0f474f6de4e0a0
Lung_Opacity-2567.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2567.png,Lung_Opacity/masks/Lung_Opacity-2567.png,True,G05144,val,e1e9da70baaa83f5a3346479a7dec18b,d16a06190b7b7f0e
Lung_Opacity-2568.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2568.png,Lung_Opacity/masks/Lung_Opacity-2568.png,True,G05145,val,77ab0dafa444bb289964ccf97dfda3ec,d56aa1155572233f
Lung_Opacity-2569.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2569.png,Lung_Opacity/masks/Lung_Opacity-2569.png,True,G05146,train,8abab779927cf713af17a24e051149fc,cd31920f1f1a4f1e
Lung_Opacity-257.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-257.png,Lung_Opacity/masks/Lung_Opacity-257.png,True,G05147,test,de236ecb334d70d255cc71fcd1df3327,9c3f430f5d0a1d4b
Lung_Opacity-2570.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2570.png,Lung_Opacity/masks/Lung_Opacity-2570.png,True,G05148,test,70e4a2b651116c9081540a41674d6498,c92d165e1e5e5878
Lung_Opacity-2571.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2571.png,Lung_Opacity/masks/Lung_Opacity-2571.png,True,G05149,train,03763312df19450b880f9e0644ecc70e,d83dd34b5e4a2a1a
Lung_Opacity-2572.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2572.png,Lung_Opacity/masks/Lung_Opacity-2572.png,True,G05150,train,b3e1a1b5a3cc6899b25896ae295d2c5f,d83953825f095b3b
Lung_Opacity-2573.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2573.png,Lung_Opacity/masks/Lung_Opacity-2573.png,True,G05151,train,e6134610de6c42d004e516e3dcd91fec,d47ec3471f4a4a0e
Lung_Opacity-2574.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2574.png,Lung_Opacity/masks/Lung_Opacity-2574.png,True,G05152,train,326cb78d46ad44d2ae24aaf878f17f86,c139525f5a0d4b1f
Lung_Opacity-2575.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2575.png,Lung_Opacity/masks/Lung_Opacity-2575.png,True,G05153,train,b2921563fc06620543d45056469a89c2,816c865f5d5a0b4f
Lung_Opacity-2576.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2576.png,Lung_Opacity/masks/Lung_Opacity-2576.png,True,G05154,test,ae45cf24834e2d2a8c4f1125b37f72ac,d02dcbcb0c0cbddc
Lung_Opacity-2577.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2577.png,Lung_Opacity/masks/Lung_Opacity-2577.png,True,G05155,train,4016b6393ed550f8a93087a996fad442,d01b9a9cc0c78fce
Lung_Opacity-2578.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2578.png,Lung_Opacity/masks/Lung_Opacity-2578.png,True,G05156,train,d8a012d8e252b3b8a115edab3d8c9e4e,c56c145a4b4b4b6f
Lung_Opacity-2579.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2579.png,Lung_Opacity/masks/Lung_Opacity-2579.png,True,G05157,train,fbedc0acbdcd8de70a78d3afd620c2ce,9939c7d65e4a4958
Lung_Opacity-258.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-258.png,Lung_Opacity/masks/Lung_Opacity-258.png,True,G05158,train,2e86d09e66a01f5c73736e83ea5ee501,ccacc61b91c39be2
Lung_Opacity-2580.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2580.png,Lung_Opacity/masks/Lung_Opacity-2580.png,True,G05159,train,1a08db780263615a7424abbb1fee2775,d46f474f2c2e1e06
Lung_Opacity-2581.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2581.png,Lung_Opacity/masks/Lung_Opacity-2581.png,True,G05160,train,933e2850488307023a6811075ac753a5,d52bf3c34b4ccc0c
Lung_Opacity-2582.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2582.png,Lung_Opacity/masks/Lung_Opacity-2582.png,True,G05161,train,ac58642d7a16d3a88a7b7dfa66d2b156,903bd6521a5b591f
Lung_Opacity-2583.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2583.png,Lung_Opacity/masks/Lung_Opacity-2583.png,True,G05162,test,a6e2cc2805323a36d176314689214d01,91685f1e1a596979
Lung_Opacity-2584.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2584.png,Lung_Opacity/masks/Lung_Opacity-2584.png,True,G05163,train,4075acc317a60f3bd2c5461ee9c149e5,dc6e4b0d450b2b3e
Lung_Opacity-2585.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2585.png,Lung_Opacity/masks/Lung_Opacity-2585.png,True,G05164,train,afbea58494feaf20ae87f6b9f93379b1,c02f420b1f4b1f6f
Lung_Opacity-2586.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2586.png,Lung_Opacity/masks/Lung_Opacity-2586.png,True,G05165,val,b29c1e8058da483cfb771870c37384d5,897917161e5f385a
Lung_Opacity-2587.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2587.png,Lung_Opacity/masks/Lung_Opacity-2587.png,True,G05166,val,8b7671da91997b42bab419ae41e0c901,83681d3c3b6b694b
Lung_Opacity-2588.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2588.png,Lung_Opacity/masks/Lung_Opacity-2588.png,True,G05167,val,77e81f439994c3c1a677a029e464f1ed,c42d0b4b4c0eaf6f
Lung_Opacity-2589.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2589.png,Lung_Opacity/masks/Lung_Opacity-2589.png,True,G05168,test,6ff1523f3a5d79f6016482d45bbbb9f5,83788c1d317a7a5b
Lung_Opacity-259.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-259.png,Lung_Opacity/masks/Lung_Opacity-259.png,True,G05169,train,4a3995be79547d4b1558fc54f164b8d6,857e06420b1f5e5f
Lung_Opacity-2590.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2590.png,Lung_Opacity/masks/Lung_Opacity-2590.png,True,G05170,test,00d7317f5c74d3dfce40bfa0db001e63,884f46186d6ea2f7
Lung_Opacity-2591.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2591.png,Lung_Opacity/masks/Lung_Opacity-2591.png,True,G05171,train,5352e41b3141d28ec7e3d449fbf80065,cc3c834b0eca6d7c
Lung_Opacity-2592.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2592.png,Lung_Opacity/masks/Lung_Opacity-2592.png,True,G05172,train,a5ae9d914d6d17dfe1a031ddd4ed73dc,9b7b041d6d3b3a12
Lung_Opacity-2593.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2593.png,Lung_Opacity/masks/Lung_Opacity-2593.png,True,G05173,train,ce2ae9ed5778fbac9188cde950e7a95c,893856d7db56054b
Lung_Opacity-2594.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2594.png,Lung_Opacity/masks/Lung_Opacity-2594.png,True,G05174,train,e758ca567423cbb1a15c871529d0495b,d52e15070f4e4f4d
Lung_Opacity-2595.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2595.png,Lung_Opacity/masks/Lung_Opacity-2595.png,True,G05175,train,266f28c9b0621899ea9d9d71df7745c8,d62f0b2d2c3f3033
Lung_Opacity-2596.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2596.png,Lung_Opacity/masks/Lung_Opacity-2596.png,True,G05176,train,4c20843f4f0176a9873f16fe354898e1,c03d87c24b1dd66b
Lung_Opacity-2597.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2597.png,Lung_Opacity/masks/Lung_Opacity-2597.png,True,G05177,train,66ce29737bc7732d6b6f4131695167c2,8939561819cff9a5
Lung_Opacity-2598.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2598.png,Lung_Opacity/masks/Lung_Opacity-2598.png,True,G05178,train,28c5612de3b119a8b056bd7d92893e37,886db7d65e484c5c
Lung_Opacity-2599.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2599.png,Lung_Opacity/masks/Lung_Opacity-2599.png,True,G05179,train,ca2b318dc5d9486d03060824802b709c,c1abd601fe114d1f
Lung_Opacity-26.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-26.png,Lung_Opacity/masks/Lung_Opacity-26.png,True,G05180,train,b4958db92d1e2d098ae7270e2c468240,9c6e061e586d7ce8
Lung_Opacity-260.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-260.png,Lung_Opacity/masks/Lung_Opacity-260.png,True,G05181,train,e1b624e2c413160fc4c494c3c7c5d69a,d02d47961be9785a
Lung_Opacity-2600.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2600.png,Lung_Opacity/masks/Lung_Opacity-2600.png,True,G05182,train,c722fe7b6731d849d7e4fdff806dadbd,dc3f41483462f3f1
Lung_Opacity-2601.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2601.png,Lung_Opacity/masks/Lung_Opacity-2601.png,True,G05183,val,ead668bc5bee38a23573ce972a239e91,906fc71a196d2e72
Lung_Opacity-2602.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2602.png,Lung_Opacity/masks/Lung_Opacity-2602.png,True,G05184,train,c7d4015af17f99b6bc0a19a3ed4ccc90,dd2d46161f483a1f
Lung_Opacity-2603.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2603.png,Lung_Opacity/masks/Lung_Opacity-2603.png,True,G05185,val,2f7df87231e49ad64aa7c2603543d52b,817b6e06fa556057
Lung_Opacity-2604.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2604.png,Lung_Opacity/masks/Lung_Opacity-2604.png,True,G05186,val,706520d1e9ee1d561186dfb04043cccb,c43f73480dc36d1d
Lung_Opacity-2605.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2605.png,Lung_Opacity/masks/Lung_Opacity-2605.png,True,G05187,test,8dec908a449496555c48ea8e003eef87,9d62965b0d1c4d1f
Lung_Opacity-2606.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2606.png,Lung_Opacity/masks/Lung_Opacity-2606.png,True,G05188,train,b8cce1f88672b33b484293253cb9d8d6,8d6c8e0e0f0f0f4f
Lung_Opacity-2607.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2607.png,Lung_Opacity/masks/Lung_Opacity-2607.png,True,G05189,test,36fce5c05f3da5836de54b2a99860ad0,c42a532e2e3b3733
Lung_Opacity-2608.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2608.png,Lung_Opacity/masks/Lung_Opacity-2608.png,True,G05190,test,bf89bbbc9f42e4b046b22db468b2bdbe,980d861e595f586f
Lung_Opacity-2609.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2609.png,Lung_Opacity/masks/Lung_Opacity-2609.png,True,G05191,train,397bc47b792671e21a92d8b60667a3e4,d129df1b3921b790
Lung_Opacity-261.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-261.png,Lung_Opacity/masks/Lung_Opacity-261.png,True,G05192,train,2d4b13513edfb680b37619fe7e2c4685,d42fdb035906f607
Lung_Opacity-2610.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2610.png,Lung_Opacity/masks/Lung_Opacity-2610.png,True,G05193,train,eb95a5ed273bd04693e4737f1216ef55,c03d53d31b1e4e1e
Lung_Opacity-2611.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2611.png,Lung_Opacity/masks/Lung_Opacity-2611.png,True,G05194,train,4f003b7b3eebce0795e2113ba1da23e8,916a5056d65a7b59
Lung_Opacity-2612.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2612.png,Lung_Opacity/masks/Lung_Opacity-2612.png,True,G05195,train,c2cdcb5f169acaaa71e96c65aa1c5670,892f960a090f5f5f
Lung_Opacity-2613.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2613.png,Lung_Opacity/masks/Lung_Opacity-2613.png,True,G05196,train,fd879a691bc0f748884438b86c133344,dd3886071b1cce5b
Lung_Opacity-2614.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2614.png,Lung_Opacity/masks/Lung_Opacity-2614.png,True,G05197,train,423a789402d328b93e61ffd39213b806,d81a5b5af8e4e4e0
Lung_Opacity-2615.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2615.png,Lung_Opacity/masks/Lung_Opacity-2615.png,True,G05198,train,74503bc72bf8c31f663e27ff33cb9eb4,913946061e5e5f5e
Lung_Opacity-2616.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2616.png,Lung_Opacity/masks/Lung_Opacity-2616.png,True,G05199,train,b46ff06fb2b71f341af8a74677b47f6c,813bd6c65e5a4a1b
Lung_Opacity-2617.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2617.png,Lung_Opacity/masks/Lung_Opacity-2617.png,True,G05200,train,95a0a58727134307a2324121b940345c,d52e42c74c0e9af9
Lung_Opacity-2618.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2618.png,Lung_Opacity/masks/Lung_Opacity-2618.png,True,G05201,val,d2729d6cc97946c6d7063d63144d8497,d42d3c53d553d251
Lung_Opacity-2619.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2619.png,Lung_Opacity/masks/Lung_Opacity-2619.png,True,G05202,train,f77ac80e9f61eff305d37eed3c85d0ba,8478ae56a23dc3d5
Lung_Opacity-262.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-262.png,Lung_Opacity/masks/Lung_Opacity-262.png,True,G05203,train,67736addbddb4c4ff2dda7a573a65130,89784e9e594a65e6
Lung_Opacity-2620.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2620.png,Lung_Opacity/masks/Lung_Opacity-2620.png,True,G05204,train,0ebab7904b47c55408a8e640b896b8f1,97694c5c5a5a6969
Lung_Opacity-2621.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2621.png,Lung_Opacity/masks/Lung_Opacity-2621.png,True,G05205,val,12450b7a8498746fac083a786f38b1c8,d2252f69f093998e
Lung_Opacity-2622.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2622.png,Lung_Opacity/masks/Lung_Opacity-2622.png,True,G05206,val,f955651244b7d6b3eab760ea279a038d,99581e59597be8e0
Lung_Opacity-2623.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2623.png,Lung_Opacity/masks/Lung_Opacity-2623.png,True,G05207,test,1fe65324836d05ae8263683dcd276a94,847e0b570b5d296d
Lung_Opacity-2624.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2624.png,Lung_Opacity/masks/Lung_Opacity-2624.png,True,G05208,test,fae28d0672479cc447fcff4345f4a741,943e87131d4f0f1d
Lung_Opacity-2625.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2625.png,Lung_Opacity/masks/Lung_Opacity-2625.png,True,G05209,test,813346ea10369cdbe8719c2576566470,d029c71b182d7b3d
Lung_Opacity-2626.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2626.png,Lung_Opacity/masks/Lung_Opacity-2626.png,True,G05210,train,e1f3bdb2281b53b01294bdbd6fc5619b,d52a820b0f1f6e6e
Lung_Opacity-2627.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2627.png,Lung_Opacity/masks/Lung_Opacity-2627.png,True,G05211,train,d5d7ae24e24268da245859f4c3f1c13e,d12985471e575755
Lung_Opacity-2628.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2628.png,Lung_Opacity/masks/Lung_Opacity-2628.png,True,G05212,train,105c73878d18131887902f4271f3e99a,817a58123b3f3bb4
Lung_Opacity-2629.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2629.png,Lung_Opacity/masks/Lung_Opacity-2629.png,True,G05213,train,81e5a40192a52de1c085ddacec4b6bf2,8e2af0979f444c1f
Lung_Opacity-263.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-263.png,Lung_Opacity/masks/Lung_Opacity-263.png,True,G05214,train,04140767946362a8980b6712d46e1361,d43d42571b4e0b3e
Lung_Opacity-2630.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2630.png,Lung_Opacity/masks/Lung_Opacity-2630.png,True,G05215,train,85f9747238e2403ddfc328dc9645ad88,884f475d6de8b4a4
Lung_Opacity-2631.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2631.png,Lung_Opacity/masks/Lung_Opacity-2631.png,True,G05216,train,7ab56e4b8e188030acb7ba47727f7104,c83f067b3564a566
Lung_Opacity-2632.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2632.png,Lung_Opacity/masks/Lung_Opacity-2632.png,True,G05217,train,ec039d927b14d4ad7c0d77972328dd78,dc6cc4d69ad5c258
Lung_Opacity-2633.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2633.png,Lung_Opacity/masks/Lung_Opacity-2633.png,True,G05218,train,99f6a84490142259caeecddbfc7cd752,852e860fbfaba1a1
Lung_Opacity-2634.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2634.png,Lung_Opacity/masks/Lung_Opacity-2634.png,True,G05219,val,017260758febad5e3d475e21ef68b5e3,916a8616cf171e4f
Lung_Opacity-2635.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2635.png,Lung_Opacity/masks/Lung_Opacity-2635.png,True,G05220,train,7c10829d342ab1223f5a0c01955f9b00,917960c6d45f965a
Lung_Opacity-2636.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2636.png,Lung_Opacity/masks/Lung_Opacity-2636.png,True,G05221,train,67c9facef93f6076f27cc1294bf9d319,946e871a0f1d2d4f
Lung_Opacity-2637.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2637.png,Lung_Opacity/masks/Lung_Opacity-2637.png,True,G05222,train,79a33a5bad6d7d682117a9a0dc3b5e18,dc3dc2431b4b2e3c
Lung_Opacity-2638.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2638.png,Lung_Opacity/masks/Lung_Opacity-2638.png,True,G05223,train,03f0238afcef5932cb95f46ffe39129d,817d86965a49655f
Lung_Opacity-2639.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2639.png,Lung_Opacity/masks/Lung_Opacity-2639.png,True,G05224,train,bf6a1061569c938da554be317f078397,da7d435b0a0b1f0e
Lung_Opacity-264.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-264.png,Lung_Opacity/masks/Lung_Opacity-264.png,True,G05225,val,9a2086088d5e5d6eaa63a2eb44440bb5,c13905465f3d5a5b
Lung_Opacity-2640.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2640.png,Lung_Opacity/masks/Lung_Opacity-2640.png,True,G05226,val,aa483bc79027b013156069b67a264672,ca3c4b0e2c0f8fb5
Lung_Opacity-2641.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2641.png,Lung_Opacity/masks/Lung_Opacity-2641.png,True,G05227,test,f0c6c43c42a8c9cf2234835dd758820c,8c66835b455f0aed
Lung_Opacity-2642.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2642.png,Lung_Opacity/masks/Lung_Opacity-2642.png,True,G05228,train,926b715f8146ca9157f3bc7638078505,c83d92474f15586f
Lung_Opacity-2643.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2643.png,Lung_Opacity/masks/Lung_Opacity-2643.png,True,G05229,test,aed8e94a9aa982558e20f1801fa3c41e,ca2fc9c54f4e0d45
Lung_Opacity-2644.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2644.png,Lung_Opacity/masks/Lung_Opacity-2644.png,True,G05230,train,306e5e396025c17e85fcc456ca56d911,c91f161a1e1a3bae
Lung_Opacity-2645.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2645.png,Lung_Opacity/masks/Lung_Opacity-2645.png,True,G05231,train,22aaf90fa49770ada123f6d76b0fd0d9,dd2857171a1d5e19
Lung_Opacity-2646.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2646.png,Lung_Opacity/masks/Lung_Opacity-2646.png,True,G05232,train,a65dd84056385e22b9c15bb5fcdc3ed4,81780e5e5b796969
Lung_Opacity-2647.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2647.png,Lung_Opacity/masks/Lung_Opacity-2647.png,True,G05233,train,cfdfa59e4afb054977bd5ab140d08e94,95686a9b69c6306f
Lung_Opacity-2648.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2648.png,Lung_Opacity/masks/Lung_Opacity-2648.png,True,G05234,test,fc12c5cbfae25db09ac08af883b3965e,913b84167b59595b
Lung_Opacity-2649.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2649.png,Lung_Opacity/masks/Lung_Opacity-2649.png,True,G05235,val,ecb99762a338eee6f76342a379e7d274,81681ec71b1bec5e
Lung_Opacity-265.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-265.png,Lung_Opacity/masks/Lung_Opacity-265.png,True,G05236,train,58b219e24277101383386618cc4712fd,dd602e4702570f5f
Lung_Opacity-2650.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2650.png,Lung_Opacity/masks/Lung_Opacity-2650.png,True,G05237,train,1e39eb5691be708d19ff5b21163643b6,cc47c34e2e2f2c6c
Lung_Opacity-2651.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2651.png,Lung_Opacity/masks/Lung_Opacity-2651.png,True,G05238,train,5df72e9ab6f2174b3d6945683d5e7dde,c52e125b3c693d36
Lung_Opacity-2652.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2652.png,Lung_Opacity/masks/Lung_Opacity-2652.png,True,G05239,train,367f85c45e149fd73f5d87875484f26c,d52e49510b4baf0f
Lung_Opacity-2653.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2653.png,Lung_Opacity/masks/Lung_Opacity-2653.png,True,G05240,train,c225df8857b674f0c3c633abcad4d02b,d53ca6f093bb0ae0
Lung_Opacity-2654.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2654.png,Lung_Opacity/masks/Lung_Opacity-2654.png,True,G05241,train,f398841648f83588d220f34e8b09da23,c93d065a1fce4ab8
Lung_Opacity-2655.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2655.png,Lung_Opacity/masks/Lung_Opacity-2655.png,True,G05242,train,caf0c10573644f6be3453e849bf2e652,812fd65a5a4f4b2a
Lung_Opacity-2656.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2656.png,Lung_Opacity/masks/Lung_Opacity-2656.png,True,G05243,train,821d75e4a7b4d496193c25b365503c64,cd3dd3435e4a4a4a
Lung_Opacity-2657.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2657.png,Lung_Opacity/masks/Lung_Opacity-2657.png,True,G05244,train,7d8c517ba8d47b964b4d7c1e67bf9be1,c57f82970b0d1d4d
Lung_Opacity-2658.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2658.png,Lung_Opacity/masks/Lung_Opacity-2658.png,True,G05245,val,8dd1bd68e2d99e4f18ec44b14e8f9c11,995e961a4d386e39
Lung_Opacity-2659.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2659.png,Lung_Opacity/masks/Lung_Opacity-2659.png,True,G05246,val,12e9bd856c528cfd35aefdab62c377f2,dd2c86530b571e0f
Lung_Opacity-266.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-266.png,Lung_Opacity/masks/Lung_Opacity-266.png,True,G05247,test,a8478817f6237584446578a78b4ed492,99ffc61d46180e1e
Lung_Opacity-2660.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2660.png,Lung_Opacity/masks/Lung_Opacity-2660.png,True,G05248,test,8a69330f991cde7237b1c9e55de74117,c02f175b2d6c7662
Lung_Opacity-2661.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2661.png,Lung_Opacity/masks/Lung_Opacity-2661.png,True,G05249,test,4adf28c45e25cfc0b1f8af722d2e84b0,837b46171b98933b
Lung_Opacity-2662.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2662.png,Lung_Opacity/masks/Lung_Opacity-2662.png,True,G05250,train,31b786628d7b7f8ea02718bdda46e200,d13bc7865a4b4b49
Lung_Opacity-2663.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2663.png,Lung_Opacity/masks/Lung_Opacity-2663.png,True,G05251,train,b379a8daeccefc831611e067bec2924a,d52a53534c4e5b3a
Lung_Opacity-2664.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2664.png,Lung_Opacity/masks/Lung_Opacity-2664.png,True,G05252,train,5596679858c836207d14c2bd22897806,d03f4a2e2d3786c6
Lung_Opacity-2665.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2665.png,Lung_Opacity/masks/Lung_Opacity-2665.png,True,G05253,train,740cdead2a98056d3ea7aacb4eafdeab,923b7b1afa13a286
Lung_Opacity-2666.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2666.png,Lung_Opacity/masks/Lung_Opacity-2666.png,True,G05254,train,484559ba2711de33d59e4bb077c2fb6b,cd2e044b5f4d6e2c
Lung_Opacity-2667.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2667.png,Lung_Opacity/masks/Lung_Opacity-2667.png,True,G05255,train,52f1d69e285786b855c90541fe44b433,c43c21cb2d9c3e9d
Lung_Opacity-2668.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2668.png,Lung_Opacity/masks/Lung_Opacity-2668.png,True,G05256,train,10838b3c77c56ccccf7abd1f4bc407ea,891d1e5b7963e4a1
Lung_Opacity-2669.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2669.png,Lung_Opacity/masks/Lung_Opacity-2669.png,True,G05257,train,8ee5905651835adec7750bc7c7b6c43c,817b101a1b7f1b3b
Lung_Opacity-267.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-267.png,Lung_Opacity/masks/Lung_Opacity-267.png,True,G05258,train,c6d43bb91d80eaf2f1213805ec91b223,985fc65818373a2f
Lung_Opacity-2670.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2670.png,Lung_Opacity/masks/Lung_Opacity-2670.png,True,G05259,train,2b05a521c5e431fc46a980d6111e9423,d4afd1554d544d54
Lung_Opacity-2671.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2671.png,Lung_Opacity/masks/Lung_Opacity-2671.png,True,G05260,train,a19027a1bb370fc366d6a88dc1611697,95681f5e78786949
Lung_Opacity-2672.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2672.png,Lung_Opacity/masks/Lung_Opacity-2672.png,True,G05261,train,aac2dda758bc111355dc5adbf80fd847,817b1e540e5e395b
Lung_Opacity-2673.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2673.png,Lung_Opacity/masks/Lung_Opacity-2673.png,True,G05262,train,518214c46759116b83622e85534e4019,917887125a4f5b5b
Lung_Opacity-2674.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2674.png,Lung_Opacity/masks/Lung_Opacity-2674.png,True,G05263,train,7cc9bf679b272e2767cb0343a3a7b55d,91794f5a1878e9e1
Lung_Opacity-2675.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2675.png,Lung_Opacity/masks/Lung_Opacity-2675.png,True,G05264,val,004e76905e14cb4f1234cd02eecf3f79,d42ddacb4f4e0e0c
Lung_Opacity-2676.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2676.png,Lung_Opacity/masks/Lung_Opacity-2676.png,True,G05265,val,30d2184646c7a41f0be937a93325fcfa,857e6f5e1c41131e
Lung_Opacity-2677.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2677.png,Lung_Opacity/masks/Lung_Opacity-2677.png,True,G05266,val,9c315b409c688057be7b8e9bcefce70f,c42afa52bf84aea4
Lung_Opacity-2678.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2678.png,Lung_Opacity/masks/Lung_Opacity-2678.png,True,G05267,test,c017d3b7533907d9e4ee655795e6b58f,c03f964a1d5c4d5d
Lung_Opacity-2679.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2679.png,Lung_Opacity/masks/Lung_Opacity-2679.png,True,G05268,test,895353a305d7d783e5650e96f4d3339c,817c065e095f2e6f
Lung_Opacity-268.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-268.png,Lung_Opacity/masks/Lung_Opacity-268.png,True,G05269,train,37f1d07201d302247f85741dd046a5cc,d53f475f1a5c2848
Lung_Opacity-2680.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2680.png,Lung_Opacity/masks/Lung_Opacity-2680.png,True,G05270,test,9fe6dce89da59026f5ae64e76150a05e,d62d510a2e2f2e6e
Lung_Opacity-2681.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2681.png,Lung_Opacity/masks/Lung_Opacity-2681.png,True,G05271,test,e40b073168fef11d6655d3ed67e53f4a,9968861e4f1e5ace
Lung_Opacity-2682.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2682.png,Lung_Opacity/masks/Lung_Opacity-2682.png,True,G05272,train,be6efba575d4b80db13d2d67c80c6e19,c03d1e075b174f1e
Lung_Opacity-2683.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2683.png,Lung_Opacity/masks/Lung_Opacity-2683.png,True,G05273,train,a52d0e1e16e5164a860529b42816e12e,803f075f1d585d5a
Lung_Opacity-2684.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2684.png,Lung_Opacity/masks/Lung_Opacity-2684.png,True,G05274,train,d7bc2f30232866af1792704e50ea8ef5,936a151e5d1f5a13
Lung_Opacity-2685.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2685.png,Lung_Opacity/masks/Lung_Opacity-2685.png,True,G05275,train,8c0e08c96fcdce18bbc5c86ac9ce6018,c13916474b1f1d3b
Lung_Opacity-2686.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2686.png,Lung_Opacity/masks/Lung_Opacity-2686.png,True,G05276,train,0848fff649bfe845e99709ca0820d289,8bf5d0b0f04f0f25
Lung_Opacity-2687.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2687.png,Lung_Opacity/masks/Lung_Opacity-2687.png,True,G05277,train,d40525d08cf2932e3f18098831d1b912,d42f050f0d5e4c5f
Lung_Opacity-2688.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2688.png,Lung_Opacity/masks/Lung_Opacity-2688.png,True,G05278,train,2b031c0ecf425743aa45402ba9d79719,cc2a929bcdccc1c7
Lung_Opacity-2689.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2689.png,Lung_Opacity/masks/Lung_Opacity-2689.png,True,G05279,train,7dbbe3f0233f041e21ae40a83ea4e01f,8d7c07171e1e1f38
Lung_Opacity-269.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-269.png,Lung_Opacity/masks/Lung_Opacity-269.png,True,G05280,train,0d005236f031a98d69ac1f9085bb2b61,ca1dc30b0f39b4ae
Lung_Opacity-2690.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2690.png,Lung_Opacity/masks/Lung_Opacity-2690.png,True,G05281,train,6f276d525dc12e027ee7e22357bf5433,907b521a393fb714
Lung_Opacity-2691.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2691.png,Lung_Opacity/masks/Lung_Opacity-2691.png,True,G05282,train,8f97fe9996f6d6404e9611746bd0e3c2,992c461ac52dbaed
Lung_Opacity-2692.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2692.png,Lung_Opacity/masks/Lung_Opacity-2692.png,True,G05283,train,46506068a250363e54241385c871f2e2,cc2ec2525f4d4e1d
Lung_Opacity-2693.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2693.png,Lung_Opacity/masks/Lung_Opacity-2693.png,True,G05284,val,0fc393ed6c571a86da2b2e663207f655,95637b786c646ca4
Lung_Opacity-2694.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2694.png,Lung_Opacity/masks/Lung_Opacity-2694.png,True,G05285,train,76656bf88d0fcb0562533a41db8d0852,91786b1f1e596949
Lung_Opacity-2695.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2695.png,Lung_Opacity/masks/Lung_Opacity-2695.png,True,G05286,val,6ff798563351c54a71525cdb7590eff2,817c17921b157c7d
Lung_Opacity-2696.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2696.png,Lung_Opacity/masks/Lung_Opacity-2696.png,True,G05287,test,3b8c6f1d3e8f95c626d24135616b8c06,813a161e1b7b3b4b
Lung_Opacity-2697.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2697.png,Lung_Opacity/masks/Lung_Opacity-2697.png,True,G05288,train,47cc8739191b68b40505879f20e33e87,817d141e191d7f4d
Lung_Opacity-2698.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2698.png,Lung_Opacity/masks/Lung_Opacity-2698.png,True,G05289,train,688574f5c0e9903485f9fb3469187643,d52a5503570f0f4f
Lung_Opacity-2699.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2699.png,Lung_Opacity/masks/Lung_Opacity-2699.png,True,G05290,val,bcd8a90f1922a78816c674a1f53a9981,99a942d239c778da
Lung_Opacity-27.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-27.png,Lung_Opacity/masks/Lung_Opacity-27.png,True,G05291,train,e2636963ebf7d7f553005e59c57c4396,dc3ad30f0c4b3d86
Lung_Opacity-270.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-270.png,Lung_Opacity/masks/Lung_Opacity-270.png,True,G05292,train,fb5159a3a982c18269155c37fb304553,d42f434f0d0d2d3d
Lung_Opacity-2700.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2700.png,Lung_Opacity/masks/Lung_Opacity-2700.png,True,G05293,test,6abe91c18818f7ee597146363f991f4a,d96a861bcd1ab8a5
Lung_Opacity-2701.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2701.png,Lung_Opacity/masks/Lung_Opacity-2701.png,True,G05294,train,629206a4e822831f17a1957b4d8b9157,dd2a035d085b5f5a
Lung_Opacity-2702.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2702.png,Lung_Opacity/masks/Lung_Opacity-2702.png,True,G05295,train,23c25cf4b7011f239c4507b60e9575e6,dc0a534b480d3faf
Lung_Opacity-2703.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2703.png,Lung_Opacity/masks/Lung_Opacity-2703.png,True,G05296,train,c7e802588f6b7831d7d7e1df40448ad6,947d63921a4b4f3c
Lung_Opacity-2704.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2704.png,Lung_Opacity/masks/Lung_Opacity-2704.png,True,G05297,train,06b4447405f82d1533b281ae1c84b851,d52e865e6c64e269
Lung_Opacity-2705.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2705.png,Lung_Opacity/masks/Lung_Opacity-2705.png,True,G05298,train,2e59fe99a11a7e7e746c5bc77ee4b41f,d1bfc617544f1c41
Lung_Opacity-2706.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2706.png,Lung_Opacity/masks/Lung_Opacity-2706.png,True,G05299,train,256c8bf4cbac5ffd9c8946b704159671,9878c74b49871f8b
Lung_Opacity-2707.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2707.png,Lung_Opacity/masks/Lung_Opacity-2707.png,True,G05300,train,49b3ba88018d71b8e5fe03794dece1d0,956b5958525bcac6
Lung_Opacity-2708.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2708.png,Lung_Opacity/masks/Lung_Opacity-2708.png,True,G05301,train,24830f46758af060abaeb0655f8caf25,dd6c47420f4d4d4d
Lung_Opacity-2709.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2709.png,Lung_Opacity/masks/Lung_Opacity-2709.png,True,G05302,train,8ba4560f520b51ab724f482d0ac1ecc8,817b565618593b3b
Lung_Opacity-271.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-271.png,Lung_Opacity/masks/Lung_Opacity-271.png,True,G05303,train,af5a9fa05afd337a03ebc30359995999,856e553e550d1d1e
Lung_Opacity-2710.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2710.png,Lung_Opacity/masks/Lung_Opacity-2710.png,True,G05304,val,b3c39b8c16b3c2642e98894fe1376251,9c6e430f0c4f4e6e
Lung_Opacity-2711.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2711.png,Lung_Opacity/masks/Lung_Opacity-2711.png,True,G05305,val,01973af39f420445f7b1cccbf06a5d87,cd2d501b0b474f3d
Lung_Opacity-2712.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2712.png,Lung_Opacity/masks/Lung_Opacity-2712.png,True,G05306,val,a2dbe2f6fc02ba0d7bf2d6dddc07bc56,9b7dc41719505753
Lung_Opacity-2713.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2713.png,Lung_Opacity/masks/Lung_Opacity-2713.png,True,G05307,test,a51b890a77273d0aa208c5db2582b339,d43c534b0a2f0b3f
Lung_Opacity-2714.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2714.png,Lung_Opacity/masks/Lung_Opacity-2714.png,True,G05308,test,bf106b293e2d976f94c4a83abcd6983b,d03d461e184d3d9f
Lung_Opacity-2715.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2715.png,Lung_Opacity/masks/Lung_Opacity-2715.png,True,G05309,test,beea29dad0a4a600146346a083a7cae4,9378c53f0f0f190d
Lung_Opacity-2716.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2716.png,Lung_Opacity/masks/Lung_Opacity-2716.png,True,G05310,train,c06f4885db679ce653797ccb2cf0c4fb,d46fc71ac83c4d0e
Lung_Opacity-2717.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2717.png,Lung_Opacity/masks/Lung_Opacity-2717.png,True,G05311,train,13e4373fe742b7edf2f27262dedaecc3,d03dd2d7521a5e43
Lung_Opacity-2718.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2718.png,Lung_Opacity/masks/Lung_Opacity-2718.png,True,G05312,test,1a7cabefdfee832d77fabfaae0110dee,dc2d420f0d4e4f6d
Lung_Opacity-2719.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2719.png,Lung_Opacity/masks/Lung_Opacity-2719.png,True,G05313,train,b7ec4c20c368b91406285b43796a3ba0,dd4e434d4d0e0e4f
Lung_Opacity-272.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-272.png,Lung_Opacity/masks/Lung_Opacity-272.png,True,G05314,train,ee2179527654c7c3be1bdad743295113,8c2ac34b6c607b7b
Lung_Opacity-2720.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2720.png,Lung_Opacity/masks/Lung_Opacity-2720.png,True,G05315,train,6184a166fe3d89afa87db5f6f322d04c,cd2213451f4d4d6f
Lung_Opacity-2721.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2721.png,Lung_Opacity/masks/Lung_Opacity-2721.png,True,G05316,train,c5c966b582b37ebbe9c547f5c2fb7f17,c42d130b5f0f6e4e
Lung_Opacity-2722.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2722.png,Lung_Opacity/masks/Lung_Opacity-2722.png,True,G05317,train,bcba21a4d9d1219ba92f14401b78ecfb,c93a170e1e1d393d
Lung_Opacity-2723.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2723.png,Lung_Opacity/masks/Lung_Opacity-2723.png,True,G05318,train,12438b40efbf1c0177a49f5c2c2802ba,d22d420e0d6d2dfd
Lung_Opacity-2724.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2724.png,Lung_Opacity/masks/Lung_Opacity-2724.png,True,G05319,train,4fb63e3b89371e8a8cfb0897c5c3c9f0,913a141f3d197b2b
Lung_Opacity-2725.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2725.png,Lung_Opacity/masks/Lung_Opacity-2725.png,True,G05320,val,a9aa2e6be03acb8177275e14279c09ac,c87d421b682c3dbd
Lung_Opacity-2726.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2726.png,Lung_Opacity/masks/Lung_Opacity-2726.png,True,G05321,val,6384bad496862a35aeeb2a7a7738e0cd,dc6dc2131b450f1f
Lung_Opacity-2727.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2727.png,Lung_Opacity/masks/Lung_Opacity-2727.png,True,G05322,test,a696501139e94c8fffbc0eccfbd5b4ae,c52e074f5e5e6868
Lung_Opacity-2728.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2728.png,Lung_Opacity/masks/Lung_Opacity-2728.png,True,G05323,test,952fc470f4c92ff325e30578f303d4ec,903b1a1a597e46ee
Lung_Opacity-2729.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2729.png,Lung_Opacity/masks/Lung_Opacity-2729.png,True,G05324,test,ab74ae0c85be47b7cf18d942bd8755e4,d5edc710d04fd04d
Lung_Opacity-273.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-273.png,Lung_Opacity/masks/Lung_Opacity-273.png,True,G05325,train,a84ad65cd624b2b7eff76fac8e8bb14e,8959061e5d5d1c7d
Lung_Opacity-2730.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2730.png,Lung_Opacity/masks/Lung_Opacity-2730.png,True,G05326,train,a4c70686cf0c3e0d481681b55cadf08c,81705f1e5e5a7a38
Lung_Opacity-2731.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2731.png,Lung_Opacity/masks/Lung_Opacity-2731.png,True,G05327,train,d0e5adae9da91b6ecd0f6f9bd8f12d41,903bc7c61cdaca9c
Lung_Opacity-2732.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2732.png,Lung_Opacity/masks/Lung_Opacity-2732.png,True,G05328,train,7a62bfcf2558eac4e4f938db97360b6a,916a071e195d595f
Lung_Opacity-2733.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2733.png,Lung_Opacity/masks/Lung_Opacity-2733.png,True,G05329,train,ef6a41187a0fed54cc574bba5dccb817,dc3ad31b4c1a4e6a
Lung_Opacity-2734.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2734.png,Lung_Opacity/masks/Lung_Opacity-2734.png,True,G05330,train,9842f07e4f916c6fd4b3da373b4c4d12,c02f5f1f5e382a31
Lung_Opacity-2735.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2735.png,Lung_Opacity/masks/Lung_Opacity-2735.png,True,G05331,train,22a75e6f7a91b6491c921bdae1b7d38a,c03d520b0f3d6d3d
Lung_Opacity-2736.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2736.png,Lung_Opacity/masks/Lung_Opacity-2736.png,True,G05332,train,836d9cdfc79585c7d233532e668ee450,997b0fc59418da9a
Lung_Opacity-2737.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2737.png,Lung_Opacity/masks/Lung_Opacity-2737.png,True,G05333,train,f5ab7bb1f81f1a917e4d6c98b29a51e8,9574070b1a2f2f1f
Lung_Opacity-2738.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2738.png,Lung_Opacity/masks/Lung_Opacity-2738.png,True,G05334,val,0b969392e38cc10ef92af38e3508c00c,c47d534c4f4d0f0e
Lung_Opacity-2739.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2739.png,Lung_Opacity/masks/Lung_Opacity-2739.png,True,G05335,train,27f7de1ed6c4e0229a850ba146e46db0,c43d534b0f0e6d2d
Lung_Opacity-274.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-274.png,Lung_Opacity/masks/Lung_Opacity-274.png,True,G05336,train,04c7743c486090969bf91edb51451188,802b7608393f7776
Lung_Opacity-2740.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2740.png,Lung_Opacity/masks/Lung_Opacity-2740.png,True,G05337,train,36bd32c5f51689a676ed1d263f3089ea,8078ff979a063e1a
Lung_Opacity-2741.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2741.png,Lung_Opacity/masks/Lung_Opacity-2741.png,True,G05338,train,bd0c9aadfdbc67c0ec944037163b34c7,9568521b5b5b1f1c
Lung_Opacity-2742.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2742.png,Lung_Opacity/masks/Lung_Opacity-2742.png,True,G05339,train,97633d1db4a08d369f4a2b24fae7beba,d83d47571e456b18
Lung_Opacity-2743.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2743.png,Lung_Opacity/masks/Lung_Opacity-2743.png,True,G05340,train,efb0e3704f82a1955deebf7b00032032,9568545b7a6e686c
Lung_Opacity-2744.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2744.png,Lung_Opacity/masks/Lung_Opacity-2744.png,True,G05341,train,5a2a05e9b4fbf9801defba8d2e8cb966,81680e1e1f1f3e5d
Lung_Opacity-2745.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2745.png,Lung_Opacity/masks/Lung_Opacity-2745.png,True,G05342,train,3035f5f37f5e5d0fd1e2ef677d68ab1b,dd2a520b0d6a2dbe
Lung_Opacity-2746.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2746.png,Lung_Opacity/masks/Lung_Opacity-2746.png,True,G05343,train,6938679ef90679429ceffbbd6d903a48,d9191e1e1a5bc6e4
Lung_Opacity-2747.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2747.png,Lung_Opacity/masks/Lung_Opacity-2747.png,True,G05344,train,e29d318ed7d62f8b450b0fade6bbb861,917f86d8ebe2a2a0
Lung_Opacity-2748.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2748.png,Lung_Opacity/masks/Lung_Opacity-2748.png,True,G05345,train,e88b6d2f6e7fe2a444ef21a187af155c,917f4c1a59ffc0c0
Lung_Opacity-2749.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2749.png,Lung_Opacity/masks/Lung_Opacity-2749.png,True,G05346,train,ae24e8588866d08360df90882ac25728,d42eb1530f4e2e1e
Lung_Opacity-275.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-275.png,Lung_Opacity/masks/Lung_Opacity-275.png,True,G05347,val,2c205c9e13949706d9391652913d2115,d42a2cc2c3d31f1f
Lung_Opacity-2750.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2750.png,Lung_Opacity/masks/Lung_Opacity-2750.png,True,G05348,val,858be298ca8cdc2d6a1ad7cb9630efc3,8179145a5b79f8a9
Lung_Opacity-2751.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2751.png,Lung_Opacity/masks/Lung_Opacity-2751.png,True,G05349,val,a7c2539446c18d5a7a2f9e7f01a42432,856f0f48496fe8b8
Lung_Opacity-2752.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2752.png,Lung_Opacity/masks/Lung_Opacity-2752.png,True,G05350,test,d19d2ca87e7dee22fa4e20ba4f8b0284,d56fd61a081c1e6d
Lung_Opacity-2753.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2753.png,Lung_Opacity/masks/Lung_Opacity-2753.png,True,G05351,test,846eb47a853d1428761bad3a3dc8becb,986e470f0f4c5c5e
Lung_Opacity-2754.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2754.png,Lung_Opacity/masks/Lung_Opacity-2754.png,True,G05352,train,6b5ac272138cf4e93ad7cd5cfed82b53,d42d07471d0f0f1f
Lung_Opacity-2755.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2755.png,Lung_Opacity/masks/Lung_Opacity-2755.png,True,G05353,train,1120843dafe5f0a8333c8a348160583f,dc2bc38ab6b2f0e0
Lung_Opacity-2756.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2756.png,Lung_Opacity/masks/Lung_Opacity-2756.png,True,G05354,train,c78946eeee7f048dc70945adfda6052b,d42f474c4e4c6af2
Lung_Opacity-2757.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2757.png,Lung_Opacity/masks/Lung_Opacity-2757.png,True,G05355,train,a9679a0dab784fa916da3173a611c4d8,ccbad355425d544b
Lung_Opacity-2758.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2758.png,Lung_Opacity/masks/Lung_Opacity-2758.png,True,G05356,train,fdbe17f6312014cd226f07f802ca2672,d43e014f0b1d2f5e
Lung_Opacity-2759.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2759.png,Lung_Opacity/masks/Lung_Opacity-2759.png,True,G05357,train,e90daf4bb4d1300d10586392ecc73d00,807f04185f5e4e5f
Lung_Opacity-276.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-276.png,Lung_Opacity/masks/Lung_Opacity-276.png,True,G05358,train,baa68482b749ae6cfa313b3ab057dd59,dc3ad3d35d1e1a08
Lung_Opacity-2760.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2760.png,Lung_Opacity/masks/Lung_Opacity-2760.png,True,G05359,train,577f64a5183744597fa00c760136fc6d,d43d43c30f3ead0a
Lung_Opacity-2761.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2761.png,Lung_Opacity/masks/Lung_Opacity-2761.png,True,G05360,train,b19553c6816cf6a332f32c6eac4c7907,dc2a43571f0a1e4f
Lung_Opacity-2762.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2762.png,Lung_Opacity/masks/Lung_Opacity-2762.png,True,G05361,train,d4718cc727946956c54284ba7ab3f18c,d1398687191fc64f
Lung_Opacity-2763.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2763.png,Lung_Opacity/masks/Lung_Opacity-2763.png,True,G05362,val,e1fb322ed964763a7b88ff63ee70824a,896d861e191f4d5d
Lung_Opacity-2764.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2764.png,Lung_Opacity/masks/Lung_Opacity-2764.png,True,G05363,train,4e44420f9f227eef4a62ff0c4018e292,d929f7c21f1e4948
Lung_Opacity-2765.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2765.png,Lung_Opacity/masks/Lung_Opacity-2765.png,True,G05364,val,2320a513d1c7199d953e8627ddfce682,d42c07c64e2d2f5b
Lung_Opacity-2766.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2766.png,Lung_Opacity/masks/Lung_Opacity-2766.png,True,G05365,train,bd81fff419f68ff46808cde690f78fb6,986c870b0f4e5b4f
Lung_Opacity-2767.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2767.png,Lung_Opacity/masks/Lung_Opacity-2767.png,True,G05366,val,d5b3726dc4a71b3c8130912fc73463b9,91794f801bede839
Lung_Opacity-2768.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2768.png,Lung_Opacity/masks/Lung_Opacity-2768.png,True,G05367,test,91774382ee51819d4bf434e5d40b43d2,97681c1e5a59595b
Lung_Opacity-2769.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2769.png,Lung_Opacity/masks/Lung_Opacity-2769.png,True,G05368,train,e3caff87540444c69190bd08940b2ae0,d42dc3171f4c6d38
Lung_Opacity-277.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-277.png,Lung_Opacity/masks/Lung_Opacity-277.png,True,G05369,test,ef7434291a538a022ed8350de0c88661,c93d86c25b1d4e4b
Lung_Opacity-2770.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2770.png,Lung_Opacity/masks/Lung_Opacity-2770.png,True,G05370,test,7d653f27d2c5501d4b6856309c4f1606,c82d17121b4b5d5f
Lung_Opacity-2771.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2771.png,Lung_Opacity/masks/Lung_Opacity-2771.png,True,G05371,train,929c9249051302c81487ce273a68316a,8370577b2b2cac6c
Lung_Opacity-2772.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2772.png,Lung_Opacity/masks/Lung_Opacity-2772.png,True,G05372,train,bbbab138f1b60a149e91ce3e5f97d591,dd2d871b0b0f0d4d
Lung_Opacity-2773.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2773.png,Lung_Opacity/masks/Lung_Opacity-2773.png,True,G05373,train,e60b21b25f25e911895a713f84746f1b,b965161b4d194d5d
Lung_Opacity-2774.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2774.png,Lung_Opacity/masks/Lung_Opacity-2774.png,True,G05374,train,cc09bdf7728a8d4975341580fa62de14,997cc6421bcf1b1c
Lung_Opacity-2775.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2775.png,Lung_Opacity/masks/Lung_Opacity-2775.png,True,G05375,train,68e32c70dd8350a3d06ac6256e12b0fe,c46ac368e0f6c49f
Lung_Opacity-2776.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2776.png,Lung_Opacity/masks/Lung_Opacity-2776.png,True,G05376,train,7a5801d8fa7c1df7b5b6f01a60348dcb,d46d071a4f4f0e4e
Lung_Opacity-2777.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2777.png,Lung_Opacity/masks/Lung_Opacity-2777.png,True,G05377,train,edfd3c62d85134f78460f38a04a5472b,d52cd24b4b4b4b0f
Lung_Opacity-2778.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2778.png,Lung_Opacity/masks/Lung_Opacity-2778.png,True,G05378,train,cd80a66c87a72aec9ef2ce8294f9c006,882f461e5d4d5c6d
Lung_Opacity-2779.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2779.png,Lung_Opacity/masks/Lung_Opacity-2779.png,True,G05379,train,774241efe038b36036173690aa1b9095,817ad4105a4b7b7b
Lung_Opacity-278.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-278.png,Lung_Opacity/masks/Lung_Opacity-278.png,True,G05380,train,ab7db8970c6697f37c49bbe251b9baba,81fc465a594a6b79
Lung_Opacity-2780.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2780.png,Lung_Opacity/masks/Lung_Opacity-2780.png,True,G05381,train,d04b028a7818cbc77806911f0e76e3d5,d969179e4f1918c9
Lung_Opacity-2781.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2781.png,Lung_Opacity/masks/Lung_Opacity-2781.png,True,G05382,test,f2fd66e16e0125aadec7d748fa137632,807fae06125e2b7d
Lung_Opacity-2782.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2782.png,Lung_Opacity/masks/Lung_Opacity-2782.png,True,G05383,train,e84b65cf756412cf36b3cd9ae0b45694,886f065b0d5d0f57
Lung_Opacity-2783.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2783.png,Lung_Opacity/masks/Lung_Opacity-2783.png,True,G05384,train,f25731313103ab544dccda37a89ac5d7,cc2f5818493cfaf4
Lung_Opacity-2784.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2784.png,Lung_Opacity/masks/Lung_Opacity-2784.png,True,G05385,val,fea4818d82238c4b2304d839e9b6bbe3,8079e7c69e87c692
Lung_Opacity-2785.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2785.png,Lung_Opacity/masks/Lung_Opacity-2785.png,True,G05386,train,b70b5be0b5f59217a2f46ae4548d634f,c42d4f0a4f4cecec
Lung_Opacity-2786.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2786.png,Lung_Opacity/masks/Lung_Opacity-2786.png,True,G05387,val,3549541b6187c4388f69a470fd156dd9,9b38ec4f781a4e58
Lung_Opacity-2787.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2787.png,Lung_Opacity/masks/Lung_Opacity-2787.png,True,G05388,val,7c31072aebc2f9a043edcc750cd95524,81389c1dfc1eea4b
Lung_Opacity-2788.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2788.png,Lung_Opacity/masks/Lung_Opacity-2788.png,True,G05389,test,1b7de23766ac886eddcdae9bccb1e70e,d12b16171a1e5a5f
Lung_Opacity-2789.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2789.png,Lung_Opacity/masks/Lung_Opacity-2789.png,True,G05390,test,552c9af575c5b0c81e7a55596a6f90a6,dd2812d30bce2f9c
Lung_Opacity-279.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-279.png,Lung_Opacity/masks/Lung_Opacity-279.png,True,G05391,train,0883d2b42f15a2efe8cd856f5c00b2c1,d42f821e4d4d7ce4
Lung_Opacity-2790.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2790.png,Lung_Opacity/masks/Lung_Opacity-2790.png,True,G05392,train,415417073660ba9aef96a10599b36c9d,817d86164f1f191f
Lung_Opacity-2791.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2791.png,Lung_Opacity/masks/Lung_Opacity-2791.png,True,G05393,train,df79db3354a2ac114eb2cd5bed0b5412,d17c12593c1a6b4f
Lung_Opacity-2792.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2792.png,Lung_Opacity/masks/Lung_Opacity-2792.png,True,G05394,train,cf3548de02c176f32d22f8a527518e07,c82e07545c6f6aea
Lung_Opacity-2793.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2793.png,Lung_Opacity/masks/Lung_Opacity-2793.png,True,G05395,train,d64f147f7cccdacbe2bf3075e23ace84,d42ec60f0aa5a2bf
Lung_Opacity-2794.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2794.png,Lung_Opacity/masks/Lung_Opacity-2794.png,True,G05396,train,9120cc97aa548aa08e2940c75571de07,94380f1f5a6d7c68
Lung_Opacity-2795.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2795.png,Lung_Opacity/masks/Lung_Opacity-2795.png,True,G05397,train,4043255f31101087502640adeedf9009,997f061869e1e4a7
Lung_Opacity-2796.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2796.png,Lung_Opacity/masks/Lung_Opacity-2796.png,True,G05398,train,bf5b3eff2202231bdc091f18c53dbec7,817b3596521b1a5f
Lung_Opacity-2797.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2797.png,Lung_Opacity/masks/Lung_Opacity-2797.png,True,G05399,train,db21949942c3ca39ea60f132e2b01d05,d43a430b0f0fad1f
Lung_Opacity-2798.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2798.png,Lung_Opacity/masks/Lung_Opacity-2798.png,True,G05400,train,87b7efb8a8a30797d7cbaf96513406e3,d0394b1f1f19391d
Lung_Opacity-2799.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2799.png,Lung_Opacity/masks/Lung_Opacity-2799.png,True,G05401,train,b7c8b2df843667e0108dc8b03cc7dbce,8931521e5e5eee38
Lung_Opacity-28.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-28.png,Lung_Opacity/masks/Lung_Opacity-28.png,True,G05402,val,eba65afb8f040d32afce5cbdf1d8840c,c0cc2fc30b3df0b3
Lung_Opacity-280.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-280.png,Lung_Opacity/masks/Lung_Opacity-280.png,True,G05403,val,343c6591df58c3dec0c9433e90854ece,817c0d1d5f597958
Lung_Opacity-2800.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2800.png,Lung_Opacity/masks/Lung_Opacity-2800.png,True,G05404,val,c5f63993cd53d7839028ecde92ef78ef,dc2d531f0d4c0e6b
Lung_Opacity-2801.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2801.png,Lung_Opacity/masks/Lung_Opacity-2801.png,True,G05405,test,f8c35e2556403c117e8738550e18ef63,d43b110d2cbdb5b1
Lung_Opacity-2802.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2802.png,Lung_Opacity/masks/Lung_Opacity-2802.png,True,G05406,test,4fb8e7334b8056460c723a4720c92d8a,98bec24772c74d49
Lung_Opacity-2803.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2803.png,Lung_Opacity/masks/Lung_Opacity-2803.png,True,G05407,train,4b840704ec2c6daa11989a095e2b70fc,d42f15038b4e6e2f
Lung_Opacity-2804.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2804.png,Lung_Opacity/masks/Lung_Opacity-2804.png,True,G05408,train,3d5a02723fcb96b2ee5197abe9558948,d53bc2171f451257
Lung_Opacity-2805.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2805.png,Lung_Opacity/masks/Lung_Opacity-2805.png,True,G05409,train,b21afd3fd1b5f3dd7ca9e43dfdc96b3d,d92d14d25b854f1b
Lung_Opacity-2806.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2806.png,Lung_Opacity/masks/Lung_Opacity-2806.png,True,G05410,test,380fd9fa4bc86993667f9a7f8c0cd35e,d02b07121d7d3d79
Lung_Opacity-2807.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2807.png,Lung_Opacity/masks/Lung_Opacity-2807.png,True,G05411,train,492ddb87f8d4c5ff1be4dd861e7867f4,913bd6103b3b3b29
Lung_Opacity-2808.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2808.png,Lung_Opacity/masks/Lung_Opacity-2808.png,True,G05412,train,28941ae54e33be5cc89f4670417f7770,d42e815f0f550b2f
Lung_Opacity-2809.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2809.png,Lung_Opacity/masks/Lung_Opacity-2809.png,True,G05413,test,2ce640303966e1e5b2e3f064e75ba683,c12f571a5c4f0d1d
Lung_Opacity-281.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-281.png,Lung_Opacity/masks/Lung_Opacity-281.png,True,G05414,train,5c9262c1f458369b6f74678c2577c92d,dc29f2d34b555650
Lung_Opacity-2810.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2810.png,Lung_Opacity/masks/Lung_Opacity-2810.png,True,G05415,train,5d53810b0097a38bcdf20b28c2e2edc2,d16dc61a19473f38
Lung_Opacity-2811.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2811.png,Lung_Opacity/masks/Lung_Opacity-2811.png,True,G05416,train,1e2e3a879a746abd92e358916c6eae90,95695e161e5ae938
Lung_Opacity-2812.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2812.png,Lung_Opacity/masks/Lung_Opacity-2812.png,True,G05417,train,9b06767f598be4ea32ede13114520e3d,99797264869b9999
Lung_Opacity-2813.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2813.png,Lung_Opacity/masks/Lung_Opacity-2813.png,True,G05418,train,5f0ee8251190937fdae9ea7af6e9a97c,817a6f5c49551d59
Lung_Opacity-2814.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2814.png,Lung_Opacity/masks/Lung_Opacity-2814.png,True,G05419,test,a3a2a01ce3efff7a598c0ee4b2108c6d,d52a520f0f3e323b
Lung_Opacity-2815.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2815.png,Lung_Opacity/masks/Lung_Opacity-2815.png,True,G05420,test,8ede44979c111baa1626d7bc8ca39f7c,d4297e1253dada52
Lung_Opacity-2816.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2816.png,Lung_Opacity/masks/Lung_Opacity-2816.png,True,G05421,train,212224ac171adf92076d75203fb29e97,d03d439b0a3d1f1e
Lung_Opacity-2817.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2817.png,Lung_Opacity/masks/Lung_Opacity-2817.png,True,G05422,train,6ce58b8c5d16c06e8b9fcd0f67929a70,8d6c07536d69e361
Lung_Opacity-2818.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2818.png,Lung_Opacity/masks/Lung_Opacity-2818.png,True,G05423,train,604854726fbe4c2bccac8781f1e3c083,8528175f5c5f2b2c
Lung_Opacity-2819.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2819.png,Lung_Opacity/masks/Lung_Opacity-2819.png,True,G05424,train,645a7d1e1784e72a0ede957543c14c1a,d42c12d34b1e6eb9
Lung_Opacity-282.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-282.png,Lung_Opacity/masks/Lung_Opacity-282.png,True,G05425,train,45b4f4e8bde33c94659eac513ba9c7ee,816bd6567a49eb09
Lung_Opacity-2820.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2820.png,Lung_Opacity/masks/Lung_Opacity-2820.png,True,G05426,train,e754177cc19211816aaa33e8d29c06a1,9b780c595b59e1e1
Lung_Opacity-2821.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2821.png,Lung_Opacity/masks/Lung_Opacity-2821.png,True,G05427,train,03528fb2b73ee38f43a1d2f2d6e2d3e0,887b05165a8f5eda
Lung_Opacity-2822.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2822.png,Lung_Opacity/masks/Lung_Opacity-2822.png,True,G05428,train,b58647f74ab43d214fe6942178150c55,885b471618eb69bd
Lung_Opacity-2823.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2823.png,Lung_Opacity/masks/Lung_Opacity-2823.png,True,G05429,train,886ba5115fc8a0efac65ac29c4bd6d85,846e474e086d6fe6
Lung_Opacity-2824.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2824.png,Lung_Opacity/masks/Lung_Opacity-2824.png,True,G05430,train,ceca8a643b008e5544f687ddb26f2521,c13b3716434f584f
Lung_Opacity-2825.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2825.png,Lung_Opacity/masks/Lung_Opacity-2825.png,True,G05431,val,7de8c734f382b7cd150a5ed3c5280e8b,917f065a5d2a672a
Lung_Opacity-2826.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2826.png,Lung_Opacity/masks/Lung_Opacity-2826.png,True,G05432,val,250be0fcfe956662b269eff50de6ee7b,d42e07454e2e2e6f
Lung_Opacity-2827.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2827.png,Lung_Opacity/masks/Lung_Opacity-2827.png,True,G05433,val,062e54dd231be618da3e2be09cc830b6,9129689e971b591f
Lung_Opacity-2828.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2828.png,Lung_Opacity/masks/Lung_Opacity-2828.png,True,G05434,train,54f58be3cc58889fbf9ed4f9ce901b4a,991e061e1a1afeea
Lung_Opacity-2829.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2829.png,Lung_Opacity/masks/Lung_Opacity-2829.png,True,G05435,train,66a317ffca79e27f180dfc5024f6b2fc,d03d151b1e1e1e5e
Lung_Opacity-283.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-283.png,Lung_Opacity/masks/Lung_Opacity-283.png,True,G05436,train,02384553019ade1fe3437ce633d99923,897bd61a1959594e
Lung_Opacity-2830.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2830.png,Lung_Opacity/masks/Lung_Opacity-2830.png,True,G05437,train,9073d3c7bfd9df23f1c5183271dbc023,886ec71b5c5c4c4f
Lung_Opacity-2831.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2831.png,Lung_Opacity/masks/Lung_Opacity-2831.png,True,G05438,train,00f09f38903ecb09fe2ba12b5411dc9e,c56f05080e2f2fed
Lung_Opacity-2832.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2832.png,Lung_Opacity/masks/Lung_Opacity-2832.png,True,G05439,train,2d38c142b68a6120dadf23e9c9b872de,c12916de4f1d1e1d
Lung_Opacity-2833.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2833.png,Lung_Opacity/masks/Lung_Opacity-2833.png,True,G05440,val,70e44dc9e23b3c3c09505a07ef96e513,937a5e7c65646445
Lung_Opacity-2834.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2834.png,Lung_Opacity/masks/Lung_Opacity-2834.png,True,G05441,val,62f233ae1ce4b91285a18df5aa8a67dd,dc2f835b055e4e64
Lung_Opacity-2835.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2835.png,Lung_Opacity/masks/Lung_Opacity-2835.png,True,G05442,test,cf15565b70a4059a14283dcd28a873f8,9f7894161a1bb91b
Lung_Opacity-2836.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2836.png,Lung_Opacity/masks/Lung_Opacity-2836.png,True,G05443,train,88894b9c835860052fd05c9e1d58554f,99090f1e797978e2
Lung_Opacity-2837.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2837.png,Lung_Opacity/masks/Lung_Opacity-2837.png,True,G05444,train,6f1afd4904d4ba090bf7e0882c68573e,c96b16161a1f5a5b
Lung_Opacity-2838.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2838.png,Lung_Opacity/masks/Lung_Opacity-2838.png,True,G05445,train,4fc7d687b3fc2eed59fb05965ac64ed9,9979c7161e471f18
Lung_Opacity-2839.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2839.png,Lung_Opacity/masks/Lung_Opacity-2839.png,True,G05446,train,b0659ab26722f7b229688ccacfd108cb,807d4583df5d095d
Lung_Opacity-284.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-284.png,Lung_Opacity/masks/Lung_Opacity-284.png,True,G05447,train,3edfb3be7c3343e7d832793dcfa60f04,dc2d470b0a0f4fab
Lung_Opacity-2840.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2840.png,Lung_Opacity/masks/Lung_Opacity-2840.png,True,G05448,train,940d141c4626c8d000c14e51578de277,987fc61e487d1c1a
Lung_Opacity-2841.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2841.png,Lung_Opacity/masks/Lung_Opacity-2841.png,True,G05449,test,9a5dba1dde6b9b53a918ccb2a0e9d46c,d42d435b0e6a2e6b
Lung_Opacity-2842.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2842.png,Lung_Opacity/masks/Lung_Opacity-2842.png,True,G05450,train,28ec2654d3e72a6e8d8f7ae0bf9402de,d42f4d4a2f2d1d0e
Lung_Opacity-2843.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2843.png,Lung_Opacity/masks/Lung_Opacity-2843.png,True,G05451,train,004974ee0589ab8888028fbd844a4030,91296a16de1fe94a
Lung_Opacity-2844.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2844.png,Lung_Opacity/masks/Lung_Opacity-2844.png,True,G05452,train,c5496da565c8b0a326258d3d0b28d0d8,dd2dd25b1e420b1d
Lung_Opacity-2845.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2845.png,Lung_Opacity/masks/Lung_Opacity-2845.png,True,G05453,val,5ffd23353ef5224d0552d31f82b72a33,996d871ac50f154f
Lung_Opacity-2846.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2846.png,Lung_Opacity/masks/Lung_Opacity-2846.png,True,G05454,test,9c4616d2dab5b8bdbfe660a9ca02d284,94674273486d6e76
Lung_Opacity-2847.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2847.png,Lung_Opacity/masks/Lung_Opacity-2847.png,True,G05455,train,6725f0cd893d595c9520a8ef2f4202f6,996b161a6978e3e1
Lung_Opacity-2848.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2848.png,Lung_Opacity/masks/Lung_Opacity-2848.png,True,G05456,train,5c72700dbaed66139b5e44b97d128b3a,9168c71e5a077a79
Lung_Opacity-2849.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2849.png,Lung_Opacity/masks/Lung_Opacity-2849.png,True,G05457,train,ce9863ad5543c443e20d3bc7a8dbd3a6,9178061e1a7b7a7a
Lung_Opacity-285.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-285.png,Lung_Opacity/masks/Lung_Opacity-285.png,True,G05458,train,7a1a326312b74c498cc951a0f2354464,dc3dcb4b486d0e2c
Lung_Opacity-2850.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2850.png,Lung_Opacity/masks/Lung_Opacity-2850.png,True,G05459,train,bf9b4e16cca3c6bff8910468fdad8da8,dc2cd1d24b4f2b0d
Lung_Opacity-2851.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2851.png,Lung_Opacity/masks/Lung_Opacity-2851.png,True,G05460,val,7e74a74635c650817f1b53139c090a7a,d57e020b1f0a3d7c
Lung_Opacity-2852.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2852.png,Lung_Opacity/masks/Lung_Opacity-2852.png,True,G05461,train,8d80d7a0bf749a71a59a45149b336449,c82d121b1f5d1b3d
Lung_Opacity-2853.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2853.png,Lung_Opacity/masks/Lung_Opacity-2853.png,True,G05462,train,6a418fced14f95bcce96f91d4655988b,817a1e181b1f5d79
Lung_Opacity-2854.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2854.png,Lung_Opacity/masks/Lung_Opacity-2854.png,True,G05463,train,3313cc0fc9e234d1941a48440bad7f14,917fae184b193959
Lung_Opacity-2855.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2855.png,Lung_Opacity/masks/Lung_Opacity-2855.png,True,G05464,train,ceddf20e3ebe9ce139a8c7cb114b1ea8,c03dcb0d2c6d3b6c
Lung_Opacity-2856.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2856.png,Lung_Opacity/masks/Lung_Opacity-2856.png,True,G05465,train,98d14febee433993eb706e16f1380388,c13e164b5e78e9a1
Lung_Opacity-2857.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2857.png,Lung_Opacity/masks/Lung_Opacity-2857.png,True,G05466,val,81116244943f2f8922d918e686048476,882f421b586f4a7f
Lung_Opacity-2858.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2858.png,Lung_Opacity/masks/Lung_Opacity-2858.png,True,G05467,train,0213a2592848fb5d9f57e65a9f01f111,c42f53534b4f4b0b
Lung_Opacity-2859.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2859.png,Lung_Opacity/masks/Lung_Opacity-2859.png,True,G05468,val,bca0c795eb38126a67d5e38ff67fc6af,dd2ad10f4c0e1b3b
Lung_Opacity-286.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-286.png,Lung_Opacity/masks/Lung_Opacity-286.png,True,G05469,test,655a167fd9b25a98360db36789236c86,913a7e952be46466
Lung_Opacity-2860.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2860.png,Lung_Opacity/masks/Lung_Opacity-2860.png,True,G05470,train,feead7b8ad71abc24f05c5c3cbfece8a,d42d4a0e2c2cfdf8
Lung_Opacity-2861.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2861.png,Lung_Opacity/masks/Lung_Opacity-2861.png,True,G05471,test,11acb89a9eb2f5f3c90dff8a2ea407bb,d52ec30b4f4e0e0f
Lung_Opacity-2862.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2862.png,Lung_Opacity/masks/Lung_Opacity-2862.png,True,G05472,test,e94674a4f86ded9a46b387c5188973aa,d03d4b4f0f091f1d
Lung_Opacity-2863.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2863.png,Lung_Opacity/masks/Lung_Opacity-2863.png,True,G05473,train,352b699ab0de7fe4b793e18a3ba883b1,d92fc61e1f021b8e
Lung_Opacity-2864.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2864.png,Lung_Opacity/masks/Lung_Opacity-2864.png,True,G05474,train,faad61cecfca140d30cc02376b8356e2,d93ac70e1d29199e
Lung_Opacity-2865.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2865.png,Lung_Opacity/masks/Lung_Opacity-2865.png,True,G05475,train,ad69eec01d95ad6fefc2097af943a000,842e0a4d5e4e6f7c
Lung_Opacity-2866.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2866.png,Lung_Opacity/masks/Lung_Opacity-2866.png,True,G05476,train,4c94a6176da59de5d808b3d9ebdbafe4,987f4b1c1c782ce5
Lung_Opacity-2867.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2867.png,Lung_Opacity/masks/Lung_Opacity-2867.png,True,G05477,train,444414f6139dc091df685b81a97fbc1f,9c768359c11d0d9f
Lung_Opacity-2868.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2868.png,Lung_Opacity/masks/Lung_Opacity-2868.png,True,G05478,train,e79ad8cf7ad7c6221d854d7cacd25456,d53be84057454b5b
Lung_Opacity-2869.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2869.png,Lung_Opacity/masks/Lung_Opacity-2869.png,True,G05479,train,ee79b33ef528b602adfac138968806e2,882f4a4a4c4f7dad
Lung_Opacity-287.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-287.png,Lung_Opacity/masks/Lung_Opacity-287.png,True,G05480,train,4a9769180a5c7239046b9dfb61aee0b6,d42f434b2d343676
Lung_Opacity-2870.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2870.png,Lung_Opacity/masks/Lung_Opacity-2870.png,True,G05481,train,942bd312c5866098361946083f1b1acb,9379141e5a787a59
Lung_Opacity-2871.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2871.png,Lung_Opacity/masks/Lung_Opacity-2871.png,True,G05482,train,a8e39bb4ea354a1319da0376a956425b,c42d13534f0e7e3c
Lung_Opacity-2872.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2872.png,Lung_Opacity/masks/Lung_Opacity-2872.png,True,G05483,train,c4e389c760e77eb73656eab540b85c82,9f685c5f484cccc5
Lung_Opacity-2873.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2873.png,Lung_Opacity/masks/Lung_Opacity-2873.png,True,G05484,val,836639c8fb05e44d5ec95312d20fddf3,917f56021f3b3839
Lung_Opacity-2874.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2874.png,Lung_Opacity/masks/Lung_Opacity-2874.png,True,G05485,val,c875dc363d37022b3ab1d75be7546e80,c97f05460c591b5f
Lung_Opacity-2875.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2875.png,Lung_Opacity/masks/Lung_Opacity-2875.png,True,G05486,train,a936810ef02ebee242d41882bccc9e39,c93c160b1f5e381f
Lung_Opacity-2876.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2876.png,Lung_Opacity/masks/Lung_Opacity-2876.png,True,G05487,val,60ed853067f2718aa9ea66a6cdceeed0,807953595b2b7939
Lung_Opacity-2877.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2877.png,Lung_Opacity/masks/Lung_Opacity-2877.png,True,G05488,test,5a876032b701fdb44e0af41f4853c148,d939165e5a1d5839
Lung_Opacity-2878.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2878.png,Lung_Opacity/masks/Lung_Opacity-2878.png,True,G05489,test,511802769823b2776f60f08b6478789c,d03d0a0f0f0d7d6d
Lung_Opacity-2879.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2879.png,Lung_Opacity/masks/Lung_Opacity-2879.png,True,G05490,train,83eac42b8ea20d181cdfb97f8f234003,d92bc656164a5d5c
Lung_Opacity-288.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-288.png,Lung_Opacity/masks/Lung_Opacity-288.png,True,G05491,train,61569cef9dfd13852044ec40852e6644,d46f434a4a0f2f2b
Lung_Opacity-2880.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2880.png,Lung_Opacity/masks/Lung_Opacity-2880.png,True,G05492,train,6829019c2b24f8e9641ecd3146283346,d6355a8a494d2f2b
Lung_Opacity-2881.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2881.png,Lung_Opacity/masks/Lung_Opacity-2881.png,True,G05493,train,f6c0cf2d2caa9e248ddfe9abb9d3ae6e,9d2ec35357452c4d
Lung_Opacity-2882.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2882.png,Lung_Opacity/masks/Lung_Opacity-2882.png,True,G05494,test,7ac5fb19c4e5424bdc559e268f5b8de0,c03d47030f4f6f0f
Lung_Opacity-2883.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2883.png,Lung_Opacity/masks/Lung_Opacity-2883.png,True,G05495,train,cc4107c7c8bc1951a22a1ac7153e49f3,c03fd75a38190b1f
Lung_Opacity-2884.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2884.png,Lung_Opacity/masks/Lung_Opacity-2884.png,True,G05496,train,314870f0523fecb61c982a1f85d2f6dd,c13f16430b1d5d4f
Lung_Opacity-2885.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2885.png,Lung_Opacity/masks/Lung_Opacity-2885.png,True,G05497,train,e3df685ee24b1b956b44b8f10766520b,d42f5e1c5f0a0f0e
Lung_Opacity-2886.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2886.png,Lung_Opacity/masks/Lung_Opacity-2886.png,True,G05498,train,f340d3798d946288e8d0c2578c9a68cb,d46d439f0e0d1f0b
Lung_Opacity-2887.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2887.png,Lung_Opacity/masks/Lung_Opacity-2887.png,True,G05499,train,fbbc1c2e93947600c4f9909d13f7f817,97684e454eccc7c5
Lung_Opacity-2888.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2888.png,Lung_Opacity/masks/Lung_Opacity-2888.png,True,G05500,train,ffb4e8282cb2bdcbdb848bb29391afe5,d83dc3531d492d1e
Lung_Opacity-2889.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2889.png,Lung_Opacity/masks/Lung_Opacity-2889.png,True,G05501,train,298a42d88ad7759e3ca4c23217011160,8179155e5b595959
Lung_Opacity-289.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-289.png,Lung_Opacity/masks/Lung_Opacity-289.png,True,G05502,train,541762548a35cec94e2fbfec7a13f746,d939c61e1c495d59
Lung_Opacity-2890.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2890.png,Lung_Opacity/masks/Lung_Opacity-2890.png,True,G05503,train,2e1e917e77501e4eb8ec4cd8782010f2,c87d420f5c492d3f
Lung_Opacity-2891.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2891.png,Lung_Opacity/masks/Lung_Opacity-2891.png,True,G05504,train,c08b0a3c7ba2b2d66db7e377f336be94,813a920a5b7f786d
Lung_Opacity-2892.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2892.png,Lung_Opacity/masks/Lung_Opacity-2892.png,True,G05505,train,adb5601acbab89f2aba48e77b9ce805d,81791f9a5b597852
Lung_Opacity-2893.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2893.png,Lung_Opacity/masks/Lung_Opacity-2893.png,True,G05506,train,8e58c950d47a12f540f4ff0f696cd77d,913bea68e1695353
Lung_Opacity-2894.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2894.png,Lung_Opacity/masks/Lung_Opacity-2894.png,True,G05507,train,9fec18f891a9737d84ec102f52db4b15,90fd4f47185d4a1d
Lung_Opacity-2895.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2895.png,Lung_Opacity/masks/Lung_Opacity-2895.png,True,G05508,train,4552dc13d10b970535fedff5e9c58795,846e07570b0f0f5f
Lung_Opacity-2896.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2896.png,Lung_Opacity/masks/Lung_Opacity-2896.png,True,G05509,val,5f01783b8e1ef5f52466a04fd367698f,ce3d406f0f4e6c0e
Lung_Opacity-2897.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2897.png,Lung_Opacity/masks/Lung_Opacity-2897.png,True,G05510,val,e9be84a4a953001df386ff790b8459e4,d03ff24b5d493918
Lung_Opacity-2898.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2898.png,Lung_Opacity/masks/Lung_Opacity-2898.png,True,G05511,test,e3c4208292c56f9cbf6f1b30339cabed,917b52183b3fbc82
Lung_Opacity-2899.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2899.png,Lung_Opacity/masks/Lung_Opacity-2899.png,True,G05512,val,b5da1a257188f6d2c2d4a212fd043f03,9d28141e585d5f5d
Lung_Opacity-29.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-29.png,Lung_Opacity/masks/Lung_Opacity-29.png,True,G05513,test,7890fe0f4035b15fc65c0edd106a462b,dc2a530d4c3d2ce7
Lung_Opacity-290.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-290.png,Lung_Opacity/masks/Lung_Opacity-290.png,True,G05514,test,57204ccef0abb6069228bd25b25b7474,dd7d46431b8d0b0b
Lung_Opacity-2900.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2900.png,Lung_Opacity/masks/Lung_Opacity-2900.png,True,G05515,train,acbab4410d7f8e60c604f19657af7ace,c03bd65a1b1a1b3b
Lung_Opacity-2901.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2901.png,Lung_Opacity/masks/Lung_Opacity-2901.png,True,G05516,test,9a7f08a01849b5215959eb70e6e64348,c07d4a420a1d7f3f
Lung_Opacity-2902.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2902.png,Lung_Opacity/masks/Lung_Opacity-2902.png,True,G05517,train,e50b27e728ad299a069bf0e709d51c34,c43dd90b0a6d2d3d
Lung_Opacity-2903.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2903.png,Lung_Opacity/masks/Lung_Opacity-2903.png,True,G05518,train,77564819421134da059c7f03d9065681,917d8616095d3f1e
Lung_Opacity-2904.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2904.png,Lung_Opacity/masks/Lung_Opacity-2904.png,True,G05519,train,6c6a0f36169c6fd9c3695abb20934f91,95681e567a6a3939
Lung_Opacity-2905.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2905.png,Lung_Opacity/masks/Lung_Opacity-2905.png,True,G05520,train,2468faa5b81c32d3fe39a71f92cd907b,d46e21c15b0def0b
Lung_Opacity-2906.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2906.png,Lung_Opacity/masks/Lung_Opacity-2906.png,True,G05521,train,3a2924a65b46e4f20339b33ff358dfb9,d47ed3d3c9d48085
Lung_Opacity-2907.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2907.png,Lung_Opacity/masks/Lung_Opacity-2907.png,True,G05522,val,4b48d048044516cc3455e28d7fad0927,d42f47565c0c4e4f
Lung_Opacity-2908.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2908.png,Lung_Opacity/masks/Lung_Opacity-2908.png,True,G05523,train,e059b70e9818aa18ddf47beb290315bb,d52bd416dc155c1d
Lung_Opacity-2909.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2909.png,Lung_Opacity/masks/Lung_Opacity-2909.png,True,G05524,train,4a248da8c15d79f96ccb939e6b60ace2,8139541f5939793d
Lung_Opacity-291.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-291.png,Lung_Opacity/masks/Lung_Opacity-291.png,True,G05525,val,4a846c847c077d042c9a57e5a7f264f8,996916181f5e195f
Lung_Opacity-2910.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2910.png,Lung_Opacity/masks/Lung_Opacity-2910.png,True,G05526,test,2af38bb0ebf08bc7da2a0193384f0c45,c03dc38b4f0f6f0a
Lung_Opacity-2911.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2911.png,Lung_Opacity/masks/Lung_Opacity-2911.png,True,G05527,train,96663a3bd3b8984800a8923d5eaf62a7,997f161b293a38a6
Lung_Opacity-2912.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2912.png,Lung_Opacity/masks/Lung_Opacity-2912.png,True,G05528,train,0dd0ce33c1e5200e4b8f227715eb52c7,c17b561658590b3f
Lung_Opacity-2913.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2913.png,Lung_Opacity/masks/Lung_Opacity-2913.png,True,G05529,train,f4be7ecac93053720b93546030648c2c,c434cb6b398e6c3c
Lung_Opacity-2914.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2914.png,Lung_Opacity/masks/Lung_Opacity-2914.png,True,G05530,train,b3ba56b099e9e9cdf84c71f728c9ab51,95685d594f69ce42
Lung_Opacity-2915.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2915.png,Lung_Opacity/masks/Lung_Opacity-2915.png,True,G05531,train,bf0920223d06ac3c0aa7db3bab31d1d3,bfe2113aba70b0a5
Lung_Opacity-2916.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2916.png,Lung_Opacity/masks/Lung_Opacity-2916.png,True,G05532,train,ffcb099136921db1de974d33467f39b0,d56b525d49091d2f
Lung_Opacity-2917.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2917.png,Lung_Opacity/masks/Lung_Opacity-2917.png,True,G05533,train,9c9c38c8fbf8e0d7843d5b641cc2425d,915a1a38e5b4d65e
Lung_Opacity-2918.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2918.png,Lung_Opacity/masks/Lung_Opacity-2918.png,True,G05534,train,ac5f55ef0127c092ec22724073e59a71,c43d574a5d2a0e57
Lung_Opacity-2919.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2919.png,Lung_Opacity/masks/Lung_Opacity-2919.png,True,G05535,val,2f3e11f4c76cc590cb4d0ec73a547dd5,917b445c1e5a1b3b
Lung_Opacity-292.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-292.png,Lung_Opacity/masks/Lung_Opacity-292.png,True,G05536,test,fc6e9f5bde1e0463d1bf13fea22d5985,8d0e074b4d5e6b6a
Lung_Opacity-2920.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2920.png,Lung_Opacity/masks/Lung_Opacity-2920.png,True,G05537,train,93249ff72be530181a9cdabcb431d5e8,c93e90431f5d4e1d
Lung_Opacity-2921.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2921.png,Lung_Opacity/masks/Lung_Opacity-2921.png,True,G05538,train,79bbfee381abe711113110a9fa31434b,dc49d30f0d1f2d2c
Lung_Opacity-2922.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2922.png,Lung_Opacity/masks/Lung_Opacity-2922.png,True,G05539,train,c9fd0be5df9fd1e9a1cf4cde8c893aba,cd2c564b4d4e495d
Lung_Opacity-2923.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2923.png,Lung_Opacity/masks/Lung_Opacity-2923.png,True,G05540,train,566499ef223cf9891db51ce7b1067281,97b1619678b1296d
Lung_Opacity-2924.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2924.png,Lung_Opacity/masks/Lung_Opacity-2924.png,True,G05541,train,3aaf3d313fcbbdebe8c8b7064245ed3d,c839131b5a393b2f
Lung_Opacity-2925.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2925.png,Lung_Opacity/masks/Lung_Opacity-2925.png,True,G05542,train,ec7364b73e4f343347b532d3e9631d9d,d53b6ac71ac818d9
Lung_Opacity-2926.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2926.png,Lung_Opacity/masks/Lung_Opacity-2926.png,True,G05543,train,542129bdeecddb6409f8d3eefdaa5dde,d52e42931b4d4f1e
Lung_Opacity-2927.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2927.png,Lung_Opacity/masks/Lung_Opacity-2927.png,True,G05544,train,4fa9067c20772927b2ab25b5397b0807,9d28065a5d6d4e7c
Lung_Opacity-2928.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2928.png,Lung_Opacity/masks/Lung_Opacity-2928.png,True,G05545,train,24c6f9eb905aefc1459c03b3cc11cc1c,c56f07494f4d49e1
Lung_Opacity-2929.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2929.png,Lung_Opacity/masks/Lung_Opacity-2929.png,True,G05546,train,992ead40f56a8543ed5c9cebfcc6db61,916ac718145f399f
Lung_Opacity-293.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-293.png,Lung_Opacity/masks/Lung_Opacity-293.png,True,G05547,train,f2bbc0283d6e1c0fec51e63341e82e0d,8370175f5e587878
Lung_Opacity-2930.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2930.png,Lung_Opacity/masks/Lung_Opacity-2930.png,True,G05548,val,3b9f221813dc21bfb495561cc5c2240d,cc18490f3e9393f3
Lung_Opacity-2931.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2931.png,Lung_Opacity/masks/Lung_Opacity-2931.png,True,G05549,val,ac8ea09d1b2edd25ffcef5c716baf53d,817c071a1f5a786f
Lung_Opacity-2932.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2932.png,Lung_Opacity/masks/Lung_Opacity-2932.png,True,G05550,val,70bf594d9301a6b61bd858a082ff3ed9,dd2ad20b4d544b4f
Lung_Opacity-2933.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2933.png,Lung_Opacity/masks/Lung_Opacity-2933.png,True,G05551,train,1d7486a5bf571d823f1fbdf705eeb9c9,817c06494d6d4f6f
Lung_Opacity-2934.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2934.png,Lung_Opacity/masks/Lung_Opacity-2934.png,True,G05552,test,065970b9792e5e86676f5c7c41da7419,cc2c53cb0e2e4caf
Lung_Opacity-2935.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2935.png,Lung_Opacity/masks/Lung_Opacity-2935.png,True,G05553,test,d603aa53be6b2ff2d1c40648986569d0,807e8e1d197376e4
Lung_Opacity-2936.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2936.png,Lung_Opacity/masks/Lung_Opacity-2936.png,True,G05554,test,88a2d2d0f8b4a4e256bbaa7e7ffcb74a,8129f2d6d017571f
Lung_Opacity-2937.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2937.png,Lung_Opacity/masks/Lung_Opacity-2937.png,True,G05555,train,483ca83f209b49f730b2dbb54898fb63,817ac652585b7b39
Lung_Opacity-2938.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2938.png,Lung_Opacity/masks/Lung_Opacity-2938.png,True,G05556,train,255c7953562fa4f93b64449464e9ae29,887dc61a1d6b1c6d
Lung_Opacity-2939.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2939.png,Lung_Opacity/masks/Lung_Opacity-2939.png,True,G05557,train,4d4c742fe4c7a0801738232522bae5cb,cd6cd0d31b51530f
Lung_Opacity-294.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-294.png,Lung_Opacity/masks/Lung_Opacity-294.png,True,G05558,train,b59304065a1ff0937b66bad325b7e153,c52897c31b1e6a5b
Lung_Opacity-2940.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2940.png,Lung_Opacity/masks/Lung_Opacity-2940.png,True,G05559,train,41788a7a900225573b293f4d430583b4,887d4a0d4c78ade7
Lung_Opacity-2941.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2941.png,Lung_Opacity/masks/Lung_Opacity-2941.png,True,G05560,train,6c2e708f2aaf8c9bfe7efac92512143f,91784ec61a38e93f
Lung_Opacity-2942.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2942.png,Lung_Opacity/masks/Lung_Opacity-2942.png,True,G05561,train,10887741cb123d8067e9c3aa1945d87e,919d4ec61e18c96f
Lung_Opacity-2943.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2943.png,Lung_Opacity/masks/Lung_Opacity-2943.png,True,G05562,train,fa9d93b7bea17e46d9a8e9dc451a4ed6,d428d10b5c2e3b3f
Lung_Opacity-2944.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2944.png,Lung_Opacity/masks/Lung_Opacity-2944.png,True,G05563,train,6e2a5d4a71de813b405fd7d33ab6829d,953c47869f1b3919
Lung_Opacity-2945.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2945.png,Lung_Opacity/masks/Lung_Opacity-2945.png,True,G05564,val,cba09ab5b91d1ef3c8fd69668c215503,917a4f143a09beb5
Lung_Opacity-2946.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2946.png,Lung_Opacity/masks/Lung_Opacity-2946.png,True,G05565,train,e890feffbd3a1ff496ca1f0a258322e7,d42fcd0e6c4c4bc3
Lung_Opacity-2947.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2947.png,Lung_Opacity/masks/Lung_Opacity-2947.png,True,G05566,val,bc11c7db0b9d228e121eb16c460fd4a4,d42f421d4c0d4d7f
Lung_Opacity-2948.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2948.png,Lung_Opacity/masks/Lung_Opacity-2948.png,True,G05567,train,412a48955aa1bdd9150a62109483287e,996c065b77726133
Lung_Opacity-2949.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2949.png,Lung_Opacity/masks/Lung_Opacity-2949.png,True,G05568,train,652eb68ec9df2159a25377423b868c8b,81fa56164e5e585d
Lung_Opacity-295.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-295.png,Lung_Opacity/masks/Lung_Opacity-295.png,True,G05569,train,c9de4add76d967d2544a33b6fbd704c8,943e6f4f09016ed5
Lung_Opacity-2950.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2950.png,Lung_Opacity/masks/Lung_Opacity-2950.png,True,G05570,train,8b514b8772fa8df5b9438b238c302bbe,807b12181b5f7e5e
Lung_Opacity-2951.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2951.png,Lung_Opacity/masks/Lung_Opacity-2951.png,True,G05571,test,a1756a62c432ac7d1090e31778d493f3,9178065e786c7979
Lung_Opacity-2952.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2952.png,Lung_Opacity/masks/Lung_Opacity-2952.png,True,G05572,train,b21e9e13e9a313a631c8cf4a2f5d50a7,d43dcb0a4e2d0d4f
Lung_Opacity-2953.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2953.png,Lung_Opacity/masks/Lung_Opacity-2953.png,True,G05573,val,e531dc8ed9aa58f8d9533b6a6c712305,c17f16065b195e1b
Lung_Opacity-2954.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2954.png,Lung_Opacity/masks/Lung_Opacity-2954.png,True,G05574,test,1d8f5af0ee44bfa449758789241ecda3,9917101d1578ffb8
Lung_Opacity-2955.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2955.png,Lung_Opacity/masks/Lung_Opacity-2955.png,True,G05575,test,ef1bc61ec3245b5591178effcb85fd95,9169c6161e0b5f3d
Lung_Opacity-2956.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2956.png,Lung_Opacity/masks/Lung_Opacity-2956.png,True,G05576,train,18b971e8d977788eb29733a69d409444,c073d4005b5b5b3f
Lung_Opacity-2957.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2957.png,Lung_Opacity/masks/Lung_Opacity-2957.png,True,G05577,test,8ce0119a99485692af1ca68b7d3de57a,91781e5a5b795383
Lung_Opacity-2958.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2958.png,Lung_Opacity/masks/Lung_Opacity-2958.png,True,G05578,test,7539dbc6a931c22bacc63535c1080dab,d07d42565ec75558
Lung_Opacity-2959.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2959.png,Lung_Opacity/masks/Lung_Opacity-2959.png,True,G05579,train,8ef3a7ce614aecd31f4011bfd539cab1,917b461e1cc8fb38
Lung_Opacity-296.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-296.png,Lung_Opacity/masks/Lung_Opacity-296.png,True,G05580,train,568c78135bcaa20fe6d006c2238c12ef,817c865f287c7a4d
Lung_Opacity-2960.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2960.png,Lung_Opacity/masks/Lung_Opacity-2960.png,True,G05581,train,01fe2f35fd0a9c83444f4b7ddf994723,8968165e1b3b593b
Lung_Opacity-2961.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2961.png,Lung_Opacity/masks/Lung_Opacity-2961.png,True,G05582,train,07ad656f34e997e6f8edcee6ea725603,d129e7969a46be14
Lung_Opacity-2962.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2962.png,Lung_Opacity/masks/Lung_Opacity-2962.png,True,G05583,train,16be8b3edd9f77fde9054eb6fa039816,9039ef5f1c494939
Lung_Opacity-2963.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2963.png,Lung_Opacity/masks/Lung_Opacity-2963.png,True,G05584,train,8d8e37a69734df18f76cc20f43720f36,dc2a73476d6530b2
Lung_Opacity-2964.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2964.png,Lung_Opacity/masks/Lung_Opacity-2964.png,True,G05585,train,ecca8d4f9d53105fdf598f252bda4805,8168021f1f5b7b5b
Lung_Opacity-2965.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2965.png,Lung_Opacity/masks/Lung_Opacity-2965.png,True,G05586,train,1d1358e941398173839ca8dbcaedd965,d52b51454a5b5a5b
Lung_Opacity-2966.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2966.png,Lung_Opacity/masks/Lung_Opacity-2966.png,True,G05587,train,c69ef2ed1436bd241566bcd8aa00b3b9,81380f085d5d5f5f
Lung_Opacity-2967.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2967.png,Lung_Opacity/masks/Lung_Opacity-2967.png,True,G05588,train,58346629132e655e41a7de35e9f240f6,d5aec25b1dc14b0d
Lung_Opacity-2968.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2968.png,Lung_Opacity/masks/Lung_Opacity-2968.png,True,G05589,train,9e636bf40769828beb9e7d5d73748129,c82ad2c715d6469f
Lung_Opacity-2969.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2969.png,Lung_Opacity/masks/Lung_Opacity-2969.png,True,G05590,test,282b8a76597f5c63c68de84da85585ef,c06daf070f470f0f
Lung_Opacity-297.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-297.png,Lung_Opacity/masks/Lung_Opacity-297.png,True,G05591,train,6d3753e6fd4263fc82391fff933602b8,913fc6161a5f1d19
Lung_Opacity-2970.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2970.png,Lung_Opacity/masks/Lung_Opacity-2970.png,True,G05592,val,723716879e71792e7bf1a075c5ff968e,887f47055d155747
Lung_Opacity-2971.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2971.png,Lung_Opacity/masks/Lung_Opacity-2971.png,True,G05593,val,7881b2450fdb00ed9032dfba63bb2ea5,cc7d534b4f0d1839
Lung_Opacity-2972.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2972.png,Lung_Opacity/masks/Lung_Opacity-2972.png,True,G05594,val,0e4fd14b83c232f5e8f5b7b339633d2f,852806171e5f7e5d
Lung_Opacity-2973.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2973.png,Lung_Opacity/masks/Lung_Opacity-2973.png,True,G05595,train,4a5a2e9ecd6ba58784f9031ba8d190c6,c93b860e191a3f3d
Lung_Opacity-2974.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2974.png,Lung_Opacity/masks/Lung_Opacity-2974.png,True,G05596,train,0c7785eb6b191d246003f0973d54b5c9,ff7fc39ad840d080
Lung_Opacity-2975.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2975.png,Lung_Opacity/masks/Lung_Opacity-2975.png,True,G05597,train,03ca8f3ad0a3d2fbecb43fe0034b63e8,992a061f5c5d5cd9
Lung_Opacity-2976.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2976.png,Lung_Opacity/masks/Lung_Opacity-2976.png,True,G05598,train,78b89a3f28f3f6dafb5d0f3f0ada4d61,9128561e1a4b5f3f
Lung_Opacity-2977.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2977.png,Lung_Opacity/masks/Lung_Opacity-2977.png,True,G05599,train,429d3712d43f379e4bd0f0e10576d082,837a04515f7b785a
Lung_Opacity-2978.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2978.png,Lung_Opacity/masks/Lung_Opacity-2978.png,True,G05600,train,607b13e103935457955814ed34646bc6,c43d1b482f3db4f0
Lung_Opacity-2979.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2979.png,Lung_Opacity/masks/Lung_Opacity-2979.png,True,G05601,train,54066281b0207efb6e71778f84ee8569,d42d530b2e6435bd
Lung_Opacity-298.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-298.png,Lung_Opacity/masks/Lung_Opacity-298.png,True,G05602,train,c1ab49db3c0d4d8ee5f2959cac363688,906fc61c38593d4f
Lung_Opacity-2980.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2980.png,Lung_Opacity/masks/Lung_Opacity-2980.png,True,G05603,train,65c2a14f5cc6207836fedd9da2b3a3bc,dc3c61430f0f6d2e
Lung_Opacity-2981.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2981.png,Lung_Opacity/masks/Lung_Opacity-2981.png,True,G05604,train,4d5b8dfcbf4953ec8d142221e3f66d08,c03d5b4b1f690d3c
Lung_Opacity-2982.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2982.png,Lung_Opacity/masks/Lung_Opacity-2982.png,True,G05605,train,ee5ea70c8949fb3ada5d08aedfeef337,9c6a431f686dbce0
Lung_Opacity-2983.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2983.png,Lung_Opacity/masks/Lung_Opacity-2983.png,True,G05606,train,4db15d5c9d92667f74b8fa78622c14b2,9968861e0e1f1b5f
Lung_Opacity-2984.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2984.png,Lung_Opacity/masks/Lung_Opacity-2984.png,True,G05607,train,3f66b341851f279631a222ea62ecceae,81fbc55a16c55a78
Lung_Opacity-2985.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2985.png,Lung_Opacity/masks/Lung_Opacity-2985.png,True,G05608,train,d50371dc70b62a7a89d681656c6413ce,81798c5e597963c6
Lung_Opacity-2986.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2986.png,Lung_Opacity/masks/Lung_Opacity-2986.png,True,G05609,train,b40bfec02c37c5d390941be73bc4e66e,c03d9783431f1e1f
Lung_Opacity-2987.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2987.png,Lung_Opacity/masks/Lung_Opacity-2987.png,True,G05610,val,c5e03c8d89ec1b38df16bd63edfe7c6f,816a1e181b5d5b5f
Lung_Opacity-2988.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2988.png,Lung_Opacity/masks/Lung_Opacity-2988.png,True,G05611,val,ee3729fd52aa3eefb3c55f7bd48f6942,817c16464f19f87d
Lung_Opacity-2989.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2989.png,Lung_Opacity/masks/Lung_Opacity-2989.png,True,G05612,val,080523a517bf8c86fac5b764da95b7ce,81798c9c5e09dc5f
Lung_Opacity-299.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-299.png,Lung_Opacity/masks/Lung_Opacity-299.png,True,G05613,test,65e73a61b8494e23c311ef5ab49093c5,d96806165b4f4c7e
Lung_Opacity-2990.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2990.png,Lung_Opacity/masks/Lung_Opacity-2990.png,True,G05614,test,4e5ee88fda48f908fe45f3c2e45debf2,891f024945fcf1f3
Lung_Opacity-2991.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2991.png,Lung_Opacity/masks/Lung_Opacity-2991.png,True,G05615,test,74cedb1dec60e7690fbabcc73f5e04c7,996f461f4d3ca138
Lung_Opacity-2992.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2992.png,Lung_Opacity/masks/Lung_Opacity-2992.png,True,G05616,train,1573621ebb952e9e681ba4a6500ef67f,887b97061e1d7e19
Lung_Opacity-2993.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2993.png,Lung_Opacity/masks/Lung_Opacity-2993.png,True,G05617,train,3cfb0622209438bf56cd58c9e1faed6e,947f030b5d4d4d6c
Lung_Opacity-2994.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2994.png,Lung_Opacity/masks/Lung_Opacity-2994.png,True,G05618,train,053485f63311ed10c5f9184ab197e33f,923d67d62cc239d8
Lung_Opacity-2995.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2995.png,Lung_Opacity/masks/Lung_Opacity-2995.png,True,G05619,train,de101628850b21d8015a7669c164025c,d43d021a4d6d6c5f
Lung_Opacity-2996.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2996.png,Lung_Opacity/masks/Lung_Opacity-2996.png,True,G05620,train,9c8a00517323bc22c6f340f71ec62f75,985945141b5f1f3b
Lung_Opacity-2997.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2997.png,Lung_Opacity/masks/Lung_Opacity-2997.png,True,G05621,train,14c11eed2a65177f5f8338fd235024ed,9168469e38eded94
Lung_Opacity-2998.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2998.png,Lung_Opacity/masks/Lung_Opacity-2998.png,True,G05622,train,09fd018350f103333a4ced8a46f2c5c4,d52a131b0d0f1f5b
Lung_Opacity-2999.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-2999.png,Lung_Opacity/masks/Lung_Opacity-2999.png,True,G05623,train,8c9eefb35bd18d607e313d9ec60089b6,d41a2f1b1d0b1f1b
Lung_Opacity-3.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3.png,Lung_Opacity/masks/Lung_Opacity-3.png,True,G05624,train,1f16201a039f37453e0968eb8af07125,9900161a5f1bff39
Lung_Opacity-30.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-30.png,Lung_Opacity/masks/Lung_Opacity-30.png,True,G05625,train,01fb532fe31d3c15ca3e04995a5d51bd,817a02495f7b7959
Lung_Opacity-300.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-300.png,Lung_Opacity/masks/Lung_Opacity-300.png,True,G05626,train,fc5f6f333593e7cc66deaa75c8f53c94,dd2957c35e505a49
Lung_Opacity-3000.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3000.png,Lung_Opacity/masks/Lung_Opacity-3000.png,True,G05627,train,efd1ea7118f0af22a6053cf28984b95c,9938a2961e8d3e9d
Lung_Opacity-3001.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3001.png,Lung_Opacity/masks/Lung_Opacity-3001.png,True,G05628,train,449b0a2fd278dceb9532137376e9b756,d46d530b0f4c0f2f
Lung_Opacity-3002.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3002.png,Lung_Opacity/masks/Lung_Opacity-3002.png,True,G05629,train,e2d213912cc86d5d70c594eebc9bc71a,897984969e1e1d3d
Lung_Opacity-3003.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3003.png,Lung_Opacity/masks/Lung_Opacity-3003.png,True,G05630,val,07977c8d2999ddb8dbedf86019561447,817916c45a5fe87a
Lung_Opacity-3004.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3004.png,Lung_Opacity/masks/Lung_Opacity-3004.png,True,G05631,val,064d3de2174aabbb70a0df771dc86305,c03d52161a5d5d5f
Lung_Opacity-3005.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3005.png,Lung_Opacity/masks/Lung_Opacity-3005.png,True,G05632,test,9e63cee20093b4e367427b81e09c3390,8029c65ea85f3e4f
Lung_Opacity-3006.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3006.png,Lung_Opacity/masks/Lung_Opacity-3006.png,True,G05633,val,c7d6bfe8d344504d2dfc69e317525fae,9178041d1d1f1f5f
Lung_Opacity-3007.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3007.png,Lung_Opacity/masks/Lung_Opacity-3007.png,True,G05634,test,b2f5b0673267745d606a1de047c33380,816e865b5d6f68a8
Lung_Opacity-3008.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3008.png,Lung_Opacity/masks/Lung_Opacity-3008.png,True,G05635,test,0a85f0878f77fead9815f904b7611cc1,d02a434b5f4e4f0f
Lung_Opacity-3009.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3009.png,Lung_Opacity/masks/Lung_Opacity-3009.png,True,G05636,train,4b6b7abf0be79992523ad4af5b2ab0a7,c97e425b0f4b0d4d
Lung_Opacity-301.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-301.png,Lung_Opacity/masks/Lung_Opacity-301.png,True,G05637,train,a61ba72d8754be6d4f1a40cf12721ac6,887d07d61a4d5e1b
Lung_Opacity-3010.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3010.png,Lung_Opacity/masks/Lung_Opacity-3010.png,True,G05638,train,8b43c6be0331b932236ca2a332bc8090,c43e07c34b6cbcc3
Lung_Opacity-3011.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3011.png,Lung_Opacity/masks/Lung_Opacity-3011.png,True,G05639,train,78745187871185ed672444b504268953,d03b5b2c24b6e666
Lung_Opacity-3012.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3012.png,Lung_Opacity/masks/Lung_Opacity-3012.png,True,G05640,train,48b64eeb39f7c1fa7574b9ade6483d3a,dd29d786cc86921d
Lung_Opacity-3013.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3013.png,Lung_Opacity/masks/Lung_Opacity-3013.png,True,G05641,train,1c9b893365bfa04f3dfe241e1d97dda3,c1195a1a1b3ba9bb
Lung_Opacity-3014.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3014.png,Lung_Opacity/masks/Lung_Opacity-3014.png,True,G05642,train,99ebe14a01dddb03642b2d56a69dbf56,8760151e171f1f5e
Lung_Opacity-3015.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3015.png,Lung_Opacity/masks/Lung_Opacity-3015.png,True,G05643,train,0b9be4839169ce3d9bbd1bf998a3f9f0,986b471e4f1c7aa8
Lung_Opacity-3016.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3016.png,Lung_Opacity/masks/Lung_Opacity-3016.png,True,G05644,train,a032252023d21a960abc623ffc610886,d42d051f0e0e2f6f
Lung_Opacity-3017.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3017.png,Lung_Opacity/masks/Lung_Opacity-3017.png,True,G05645,train,b30a8a9fefa5b7ec02daebf1a63cd565,d42fc2c34b4b495d
Lung_Opacity-3018.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3018.png,Lung_Opacity/masks/Lung_Opacity-3018.png,True,G05646,train,67844c7ab18fddff22bf64da2d467ed0,c6228345272e6f6f
Lung_Opacity-3019.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3019.png,Lung_Opacity/masks/Lung_Opacity-3019.png,True,G05647,test,ea1a601cdc8ffd669bdfac724239ebe2,9d6c871b52475b4a
Lung_Opacity-302.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-302.png,Lung_Opacity/masks/Lung_Opacity-302.png,True,G05648,train,d0dd969f1433cba6ee9b4229085fa573,dc3d430b4d2a3d2d
Lung_Opacity-3020.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3020.png,Lung_Opacity/masks/Lung_Opacity-3020.png,True,G05649,val,606c4fc5a5f1edc3b5d4c4d962f99d3c,8b680c9e1a7a7979
Lung_Opacity-3021.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3021.png,Lung_Opacity/masks/Lung_Opacity-3021.png,True,G05650,train,4c441ff3e80b464976dee6982493053a,cc7f525b0949192f
Lung_Opacity-3022.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3022.png,Lung_Opacity/masks/Lung_Opacity-3022.png,True,G05651,val,f3ae874e3c39c52375b465cd1cb681a2,956e031e5a7d6865
Lung_Opacity-3023.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3023.png,Lung_Opacity/masks/Lung_Opacity-3023.png,True,G05652,train,7136a07a451abf19808a31d5354b8b64,984e0e1a6979ebb1
Lung_Opacity-3024.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3024.png,Lung_Opacity/masks/Lung_Opacity-3024.png,True,G05653,val,b44c107e475d21fb2391d593c759fc83,c42d031a0d2fed6f
Lung_Opacity-3025.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3025.png,Lung_Opacity/masks/Lung_Opacity-3025.png,True,G05654,test,c6cd2a2ffeaaeebde77a68485c4cfae2,81681f9e1f5d3839
Lung_Opacity-3026.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3026.png,Lung_Opacity/masks/Lung_Opacity-3026.png,True,G05655,test,30f97f56394bbcab3ea871d1a82a7a30,cc2fd31b5c58494d
Lung_Opacity-3027.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3027.png,Lung_Opacity/masks/Lung_Opacity-3027.png,True,G05656,train,a27c06550921b00d2937fdb21dded195,cd2e8b4f2a52d2d2
Lung_Opacity-3028.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3028.png,Lung_Opacity/masks/Lung_Opacity-3028.png,True,G05657,train,4bcb90c4b9decfacd17dbb19cf08eb28,dd2f42461f5b492c
Lung_Opacity-3029.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3029.png,Lung_Opacity/masks/Lung_Opacity-3029.png,True,G05658,train,06e53e86518926096d84e78a658fbe1f,897d141b7959595a
Lung_Opacity-303.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-303.png,Lung_Opacity/masks/Lung_Opacity-303.png,True,G05659,train,ee9f67b4663e05c2e5912d33305d4d22,987bc5061e59591f
Lung_Opacity-3030.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3030.png,Lung_Opacity/masks/Lung_Opacity-3030.png,True,G05660,train,c18186414ec262a8eab32d252f14f2d3,d03d4b461a392dbd
Lung_Opacity-3031.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3031.png,Lung_Opacity/masks/Lung_Opacity-3031.png,True,G05661,train,f2262d7e1546a44a2fe3cd9b8d0ef600,d93bd5061e38d90b
Lung_Opacity-3032.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3032.png,Lung_Opacity/masks/Lung_Opacity-3032.png,True,G05662,train,a77399b9d3b945d16075276ad79ac44c,817a501f1b3b3a3e
Lung_Opacity-3033.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3033.png,Lung_Opacity/masks/Lung_Opacity-3033.png,True,G05663,train,35623e11b02327b664025bdc389753d8,c16f070e1b0b6f4b
Lung_Opacity-3034.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3034.png,Lung_Opacity/masks/Lung_Opacity-3034.png,True,G05664,train,5b059b5f267eb6f839353b92fe0a9d2b,c43c8f831f4d4d3c
Lung_Opacity-3035.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3035.png,Lung_Opacity/masks/Lung_Opacity-3035.png,True,G05665,train,7a097d58ac8f4c11d9594c1ece9e2a43,9179e147aa9e9686
Lung_Opacity-3036.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3036.png,Lung_Opacity/masks/Lung_Opacity-3036.png,True,G05666,train,86e92fcd515dfda843efe4f615b7d11b,d85b05161e5b5f1a
Lung_Opacity-3037.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3037.png,Lung_Opacity/masks/Lung_Opacity-3037.png,True,G05667,train,72b38451fa913d71e80952bc9bef85bf,c02fe44e4b692d6d
Lung_Opacity-3038.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3038.png,Lung_Opacity/masks/Lung_Opacity-3038.png,True,G05668,train,1fa805517d755ce5fa2f8662af176beb,d52e444f4e297d61
Lung_Opacity-3039.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3039.png,Lung_Opacity/masks/Lung_Opacity-3039.png,True,G05669,train,eaba127ff07fe0d8252411fbf2f1e85c,d13bc21a39293d3e
Lung_Opacity-304.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-304.png,Lung_Opacity/masks/Lung_Opacity-304.png,True,G05670,val,c36ca72b0118bd19fbbaabdfa9f799c4,f97f965a5958180a
Lung_Opacity-3040.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3040.png,Lung_Opacity/masks/Lung_Opacity-3040.png,True,G05671,val,c4a223bd56de1896e93cc1a88742436e,c42e070e0e6a6f7d
Lung_Opacity-3041.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3041.png,Lung_Opacity/masks/Lung_Opacity-3041.png,True,G05672,val,9dfe999f5b81789a26867d5aaf741075,c83d431b1bcf1f48
Lung_Opacity-3042.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3042.png,Lung_Opacity/masks/Lung_Opacity-3042.png,True,G05673,test,3a7a991b5bc6d2dfc6a31642601cb151,d42fc25f97433c41
Lung_Opacity-3043.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3043.png,Lung_Opacity/masks/Lung_Opacity-3043.png,True,G05674,train,114ee7f3abd1caa5ace4d5adf41522a9,d43d4b091c4bb4bb
Lung_Opacity-3044.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3044.png,Lung_Opacity/masks/Lung_Opacity-3044.png,True,G05675,test,c792bc20d409e143f9744d34a2d95e96,916e471a58e1bcce
Lung_Opacity-3045.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3045.png,Lung_Opacity/masks/Lung_Opacity-3045.png,True,G05676,test,27a9d7fb1d2db6780f56463767cad244,9c6e451f4c5c6c4e
Lung_Opacity-3046.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3046.png,Lung_Opacity/masks/Lung_Opacity-3046.png,True,G05677,train,df0a684739eb8c1a3df2217f5fcd10b3,d52b43570d0d5c5d
Lung_Opacity-3047.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3047.png,Lung_Opacity/masks/Lung_Opacity-3047.png,True,G05678,train,276b1c7169c2112904a6eb59c705e219,917a861a191d5f4f
Lung_Opacity-3048.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3048.png,Lung_Opacity/masks/Lung_Opacity-3048.png,True,G05679,train,15cca3007d1ad7154478ae4fcd20baeb,d52e93491e2e7233
Lung_Opacity-3049.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3049.png,Lung_Opacity/masks/Lung_Opacity-3049.png,True,G05680,train,2413b5f44eda92a3c865b22e481f3f65,817f0718494d7c5f
Lung_Opacity-305.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-305.png,Lung_Opacity/masks/Lung_Opacity-305.png,True,G05681,train,cd9b4337d34b8ebb91aaa7c884c1af02,d06d454f4e1f194d
Lung_Opacity-3050.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3050.png,Lung_Opacity/masks/Lung_Opacity-3050.png,True,G05682,train,162df812cf993bd01369e5f23d1ddaf3,c72dd8440b4f6f0b
Lung_Opacity-3051.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3051.png,Lung_Opacity/masks/Lung_Opacity-3051.png,True,G05683,train,6c183fbefcdbc7266fe6177979c552dd,d42d13d3cb0e1e53
Lung_Opacity-3052.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3052.png,Lung_Opacity/masks/Lung_Opacity-3052.png,True,G05684,train,d23a4e99769a28b47e723cd9666537f3,812e071e1f1f1d4f
Lung_Opacity-3053.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3053.png,Lung_Opacity/masks/Lung_Opacity-3053.png,True,G05685,val,d7c789e25369ef566fb8b97f655c8f6c,8139061e5d5e7b39
Lung_Opacity-3054.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3054.png,Lung_Opacity/masks/Lung_Opacity-3054.png,True,G05686,val,9842a8fbedf474971ce8e4b66c93be91,991b161a39fbe0e1
Lung_Opacity-3055.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3055.png,Lung_Opacity/masks/Lung_Opacity-3055.png,True,G05687,train,2650fbde2fae1fe5dbe77a6ef9193def,d03952835a1f2e5f
Lung_Opacity-3056.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3056.png,Lung_Opacity/masks/Lung_Opacity-3056.png,True,G05688,train,fc76cca6a982d565b2d999c896004eac,9c2a835b5c4c78fc
Lung_Opacity-3057.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3057.png,Lung_Opacity/masks/Lung_Opacity-3057.png,True,G05689,train,c0e222c1b94ccb70e30e74f599c9ea72,80795a0b39fdb1b1
Lung_Opacity-3058.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3058.png,Lung_Opacity/masks/Lung_Opacity-3058.png,True,G05690,train,31cf2f093d585f747b23e47663ed168e,c42e37d3c34f0e1c
Lung_Opacity-3059.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3059.png,Lung_Opacity/masks/Lung_Opacity-3059.png,True,G05691,train,8dd584ccb7a49649e301db049cd1b3ad,c97d161e5d19494d
Lung_Opacity-306.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-306.png,Lung_Opacity/masks/Lung_Opacity-306.png,True,G05692,train,6546ace99da99c7cec7883e143693c01,9c2c460b59ececbc
Lung_Opacity-3060.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3060.png,Lung_Opacity/masks/Lung_Opacity-3060.png,True,G05693,train,5fec2b5b205a79ac35cd5725f89d4f86,946c464f4c1f4e5d
Lung_Opacity-3061.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3061.png,Lung_Opacity/masks/Lung_Opacity-3061.png,True,G05694,train,940ba396ccb4ad10bf4244594d898301,dd2cc34b1d0e4a3e
Lung_Opacity-3062.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3062.png,Lung_Opacity/masks/Lung_Opacity-3062.png,True,G05695,val,6441ec85246e7e6b068f8e77c47fe9de,df5fe0e05a78e082
Lung_Opacity-3063.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3063.png,Lung_Opacity/masks/Lung_Opacity-3063.png,True,G05696,test,23d3d9e41f2a324030ce7b0c0add368c,c03d57430b4f3c1f
Lung_Opacity-3064.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3064.png,Lung_Opacity/masks/Lung_Opacity-3064.png,True,G05697,test,7bfe3dc8a39bab377e9a328a917319ff,85781f1e5a58787a
Lung_Opacity-3065.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3065.png,Lung_Opacity/masks/Lung_Opacity-3065.png,True,G05698,test,3a50ef530d22d51c9a7671f83af8daf8,896f865a5b4d7a28
Lung_Opacity-3066.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3066.png,Lung_Opacity/masks/Lung_Opacity-3066.png,True,G05699,train,6f21a8f54200c2234f7924baf8127017,d42a4b4d0e2f1d2f
Lung_Opacity-3067.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3067.png,Lung_Opacity/masks/Lung_Opacity-3067.png,True,G05700,train,fc88aaf5bba79075964eea7e49e123c0,d03b521a1812bfbf
Lung_Opacity-3068.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3068.png,Lung_Opacity/masks/Lung_Opacity-3068.png,True,G05701,train,0fef04fb90e3da6c998b1a4faed4839b,d42a974b0e2ce6d3
Lung_Opacity-3069.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3069.png,Lung_Opacity/masks/Lung_Opacity-3069.png,True,G05702,train,38890d60c4f6b85fdb75b542fd01d3c4,957e074d5c6c686c
Lung_Opacity-307.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-307.png,Lung_Opacity/masks/Lung_Opacity-307.png,True,G05703,val,7b641e19ed766003c08f03ffa3132991,dc6a530b6b0f2c2d
Lung_Opacity-3070.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3070.png,Lung_Opacity/masks/Lung_Opacity-3070.png,True,G05704,train,74bef451e15f51d30451c4d35667db55,cc3d52430f9c68af
Lung_Opacity-3071.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3071.png,Lung_Opacity/masks/Lung_Opacity-3071.png,True,G05705,train,5298a38b71a1fea90a1d63ad4ff2a753,c13d101a5f4b7979
Lung_Opacity-3072.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3072.png,Lung_Opacity/masks/Lung_Opacity-3072.png,True,G05706,train,2033e343738bd455285da7476a1c8070,883f061e185d5f7c
Lung_Opacity-3073.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3073.png,Lung_Opacity/masks/Lung_Opacity-3073.png,True,G05707,train,f460269eb1992e79535a92c492bf5d80,85654751793a753a
Lung_Opacity-3074.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3074.png,Lung_Opacity/masks/Lung_Opacity-3074.png,True,G05708,train,bd75a6508e752e850cae04dd4ad1eaf5,d43ee70ac04b5b4b
Lung_Opacity-3075.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3075.png,Lung_Opacity/masks/Lung_Opacity-3075.png,True,G05709,train,249b4feaba98b803bd9231a497a06b19,c97d925a144b3d4b
Lung_Opacity-3076.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3076.png,Lung_Opacity/masks/Lung_Opacity-3076.png,True,G05710,test,5517597bb267b82dc22586787917e129,916a1edf39e524c4
Lung_Opacity-3077.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3077.png,Lung_Opacity/masks/Lung_Opacity-3077.png,True,G05711,train,521230d2f673f658ea3a191199c33076,d53f07120f5a585d
Lung_Opacity-3078.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3078.png,Lung_Opacity/masks/Lung_Opacity-3078.png,True,G05712,test,ef05b362581a0f7ed4dc0f2b6a77e974,d43f434b4f4f0c4c
Lung_Opacity-3079.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3079.png,Lung_Opacity/masks/Lung_Opacity-3079.png,True,G05713,train,5a5f3596bcc4c92102f520a3e77d4400,817b14165a5b3b3b
Lung_Opacity-308.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-308.png,Lung_Opacity/masks/Lung_Opacity-308.png,True,G05714,val,de260e60237025b5b975a761410f091c,d42e83572c26666f
Lung_Opacity-3080.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3080.png,Lung_Opacity/masks/Lung_Opacity-3080.png,True,G05715,val,d968420b1136f9ddddf6b51144a2742a,946e435f4c0e4e4f
Lung_Opacity-3081.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3081.png,Lung_Opacity/masks/Lung_Opacity-3081.png,True,G05716,test,57c44e5936b92aae0c771320702e9afa,c46f42184f4f4d6d
Lung_Opacity-3082.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3082.png,Lung_Opacity/masks/Lung_Opacity-3082.png,True,G05717,val,3cf9094bc50d12f18355e2d775e0059b,c42e53586c2a3deb
Lung_Opacity-3083.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3083.png,Lung_Opacity/masks/Lung_Opacity-3083.png,True,G05718,train,04b593f4087edc1145851dd7eccbd76f,997896185d7a784e
Lung_Opacity-3084.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3084.png,Lung_Opacity/masks/Lung_Opacity-3084.png,True,G05719,train,369b20d8acf148b2e566e76fff2962a7,8c2e83d30f460f5f
Lung_Opacity-3085.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3085.png,Lung_Opacity/masks/Lung_Opacity-3085.png,True,G05720,train,8bcd726845461c80147443cb6095ae7a,897b0596161a5f5e
Lung_Opacity-3086.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3086.png,Lung_Opacity/masks/Lung_Opacity-3086.png,True,G05721,train,05cd142433ba2a7ae4128ec401af77e1,897d065a696762e6
Lung_Opacity-3087.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3087.png,Lung_Opacity/masks/Lung_Opacity-3087.png,True,G05722,val,0317d6a68c318afae9206533dcc37c51,89381e5a5b5a5e72
Lung_Opacity-3088.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3088.png,Lung_Opacity/masks/Lung_Opacity-3088.png,True,G05723,test,8dda78de288e26275e5a2e552063ea62,93788c103e7e7939
Lung_Opacity-3089.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3089.png,Lung_Opacity/masks/Lung_Opacity-3089.png,True,G05724,train,31195d60db762aff31091e89676b037e,95684d5b5b5dce40
Lung_Opacity-309.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-309.png,Lung_Opacity/masks/Lung_Opacity-309.png,True,G05725,train,d411088360406392e6179b4a13839074,8c6d170a4a7c6e6e
Lung_Opacity-3090.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3090.png,Lung_Opacity/masks/Lung_Opacity-3090.png,True,G05726,train,088171ea09974e80e888acbca826d9ec,d029735d1c39399b
Lung_Opacity-3091.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3091.png,Lung_Opacity/masks/Lung_Opacity-3091.png,True,G05727,test,2bdccfb64d3c51d5c00aab1a83a719f1,847e0f510f4b0f5b
Lung_Opacity-3092.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3092.png,Lung_Opacity/masks/Lung_Opacity-3092.png,True,G05728,train,3cdb02015f1b84fa8d66f04a9bf63b09,d42fe335fd42510a
Lung_Opacity-3093.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3093.png,Lung_Opacity/masks/Lung_Opacity-3093.png,True,G05729,test,aa8566ca999eb843f6a4c560750e915b,9959c61e58495b5d
Lung_Opacity-3094.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3094.png,Lung_Opacity/masks/Lung_Opacity-3094.png,True,G05730,val,b1f82b5840b78f55bf6ef5982f007e69,d12e4e0e194dcf59
Lung_Opacity-3095.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3095.png,Lung_Opacity/masks/Lung_Opacity-3095.png,True,G05731,train,e39fc12a3dcda13c4ded8851fe52f175,8831469e5a5f39cd
Lung_Opacity-3096.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3096.png,Lung_Opacity/masks/Lung_Opacity-3096.png,True,G05732,train,0b4ce7d52be7359d2fe5c5c08d743182,8c2a47cb1d5d0d5e
Lung_Opacity-3097.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3097.png,Lung_Opacity/masks/Lung_Opacity-3097.png,True,G05733,test,42fd04f11106255415aa8156b5a15f62,816cc756894f396d
Lung_Opacity-3098.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3098.png,Lung_Opacity/masks/Lung_Opacity-3098.png,True,G05734,train,97c79eac97a6eca672483cb2c6a63491,8d2cc6564f460f5b
Lung_Opacity-3099.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3099.png,Lung_Opacity/masks/Lung_Opacity-3099.png,True,G05735,train,ce42492235d7e91d09abd3de06fcf937,857c0e5f1b19786c
Lung_Opacity-31.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-31.png,Lung_Opacity/masks/Lung_Opacity-31.png,True,G05736,train,3245a53ee3e95c7386ef5a982dc2bf69,982a47565d4d69bc
Lung_Opacity-310.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-310.png,Lung_Opacity/masks/Lung_Opacity-310.png,True,G05737,train,0366d48c6a1711fd560efe27849cd7d0,916847161a6f393f
Lung_Opacity-3100.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3100.png,Lung_Opacity/masks/Lung_Opacity-3100.png,True,G05738,val,af76138d56aee5d254e12383ccfcb4ef,9f60071e1c5c3c77
Lung_Opacity-3101.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3101.png,Lung_Opacity/masks/Lung_Opacity-3101.png,True,G05739,train,beb6d3a3dcb6fca6880332238b0f7d72,c83d1b0b2d2dbe15
Lung_Opacity-3102.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3102.png,Lung_Opacity/masks/Lung_Opacity-3102.png,True,G05740,train,da7a907e545d10ab19bb2540429903da,c02f52513d2f3736
Lung_Opacity-3103.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3103.png,Lung_Opacity/masks/Lung_Opacity-3103.png,True,G05741,train,7450e24a9e7193506d9792efb6545ba3,cd2e064f6c64e3a3
Lung_Opacity-3104.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3104.png,Lung_Opacity/masks/Lung_Opacity-3104.png,True,G05742,train,e53a3701a4b9d82a9e8ef315cd901abf,895f061e5b6928de
Lung_Opacity-3105.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3105.png,Lung_Opacity/masks/Lung_Opacity-3105.png,True,G05743,train,0f8c1194756111a8536b101fa996b792,d038475b1b59795c
Lung_Opacity-3106.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3106.png,Lung_Opacity/masks/Lung_Opacity-3106.png,True,G05744,train,56c6c3c61dd6bd877a4131d34c7f7071,d17f86165f1b090d
Lung_Opacity-3107.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3107.png,Lung_Opacity/masks/Lung_Opacity-3107.png,True,G05745,train,4f3ef21851a9ef2661685762c3696141,de2d71434b4d4c4e
Lung_Opacity-3108.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3108.png,Lung_Opacity/masks/Lung_Opacity-3108.png,True,G05746,train,73abd1fe1af7b7fcb56006e5e2bfbb67,d42f81414f2e7f2a
Lung_Opacity-3109.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3109.png,Lung_Opacity/masks/Lung_Opacity-3109.png,True,G05747,train,e78e105f2c80f511468c6abad1b1ba35,d12b54105b5d5f0f
Lung_Opacity-311.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-311.png,Lung_Opacity/masks/Lung_Opacity-311.png,True,G05748,train,bc79c858cb465754b55113cb1e3195fb,85695e1e4a39e1e5
Lung_Opacity-3110.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3110.png,Lung_Opacity/masks/Lung_Opacity-3110.png,True,G05749,train,b81b4b7387e6a57a1d86eb79fa27c349,907a5a181f1fbaaa
Lung_Opacity-3111.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3111.png,Lung_Opacity/masks/Lung_Opacity-3111.png,True,G05750,train,5378b4db2a9202d222c08171e1faf9ad,c43e874b0e6c6c6e
Lung_Opacity-3112.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3112.png,Lung_Opacity/masks/Lung_Opacity-3112.png,True,G05751,train,d242460e37e629feff6055820ed86469,807b671e5e563859
Lung_Opacity-3113.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3113.png,Lung_Opacity/masks/Lung_Opacity-3113.png,True,G05752,val,e6d535b47bfc61920b47685b405892d9,d025d3de1e68383d
Lung_Opacity-3114.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3114.png,Lung_Opacity/masks/Lung_Opacity-3114.png,True,G05753,val,936ac14e716a27817270af37cd8b2263,986d071f0b1d4f1e
Lung_Opacity-3115.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3115.png,Lung_Opacity/masks/Lung_Opacity-3115.png,True,G05754,test,98170fd293916167aeed1dc4a507ab98,dd2a431f18193dab
Lung_Opacity-3116.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3116.png,Lung_Opacity/masks/Lung_Opacity-3116.png,True,G05755,train,a2766f0875850f673724486340cb3f0e,d43f42571b4a4b1d
Lung_Opacity-3117.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3117.png,Lung_Opacity/masks/Lung_Opacity-3117.png,True,G05756,test,74995b2f22af071c49cb15eeccbca870,805b1f5a5b785b68
Lung_Opacity-3118.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3118.png,Lung_Opacity/masks/Lung_Opacity-3118.png,True,G05757,train,33accb70bea3e0870611687cf6a9ccf4,d42c51171f2f0f0f
Lung_Opacity-3119.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3119.png,Lung_Opacity/masks/Lung_Opacity-3119.png,True,G05758,train,c0b65a800d3ea7334dab37a4fa040ba5,992a0d19297dadad
Lung_Opacity-312.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-312.png,Lung_Opacity/masks/Lung_Opacity-312.png,True,G05759,train,0a1341f0fd240e2e26372c7bbe196285,d93bd75059585919
Lung_Opacity-3120.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3120.png,Lung_Opacity/masks/Lung_Opacity-3120.png,True,G05760,train,f1f75d069c725969eb7746dd8a5c7b90,d42f550f4f0a0f2e
Lung_Opacity-3121.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3121.png,Lung_Opacity/masks/Lung_Opacity-3121.png,True,G05761,train,52922d01e6b6d7761ebd54dfefe738a0,d911161e1a1b3b3f
Lung_Opacity-3122.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3122.png,Lung_Opacity/masks/Lung_Opacity-3122.png,True,G05762,train,0eb807a2b6add35036d2131ff2564323,802f468d5e4c7c5f
Lung_Opacity-3123.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3123.png,Lung_Opacity/masks/Lung_Opacity-3123.png,True,G05763,train,67b4e1fa31e1c7c38676e0f84d275266,995bc61c584b0bb7
Lung_Opacity-3124.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3124.png,Lung_Opacity/masks/Lung_Opacity-3124.png,True,G05764,train,c292fd39fcafefc70ba71ca5d15aca7d,d42e07135e7e6c68
Lung_Opacity-3125.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3125.png,Lung_Opacity/masks/Lung_Opacity-3125.png,True,G05765,train,6d8ee6f5a822bef31065b9d5dc94c881,803fd656595c5359
Lung_Opacity-3126.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3126.png,Lung_Opacity/masks/Lung_Opacity-3126.png,True,G05766,train,5a494e051e5c43e38a5e67bb4c93fe99,94295f6a67623167
Lung_Opacity-3127.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3127.png,Lung_Opacity/masks/Lung_Opacity-3127.png,True,G05767,train,9cdaef2276b64f94dbd8eee43afbd731,80c46bca3c8f6d37
Lung_Opacity-3128.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3128.png,Lung_Opacity/masks/Lung_Opacity-3128.png,True,G05768,val,0df72e7a377217860ee24ff681d35919,997b4716184e1fb8
Lung_Opacity-3129.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3129.png,Lung_Opacity/masks/Lung_Opacity-3129.png,True,G05769,train,251049fc6f60edcb0b65c8d5af1d3c30,d13b160e1a4c79bb
Lung_Opacity-313.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-313.png,Lung_Opacity/masks/Lung_Opacity-313.png,True,G05770,val,e030d3bcf9e4c4955656fd0c69f97bc0,d46eb7530b4f2c0c
Lung_Opacity-3130.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3130.png,Lung_Opacity/masks/Lung_Opacity-3130.png,True,G05771,train,6724e042e11f1e3c87a9ebe4587f2b81,91681f5a5949e9cd
Lung_Opacity-3131.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3131.png,Lung_Opacity/masks/Lung_Opacity-3131.png,True,G05772,test,e74e0698c07aba50143ef36860c79e21,887f875e1c155b19
Lung_Opacity-3132.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3132.png,Lung_Opacity/masks/Lung_Opacity-3132.png,True,G05773,train,a4e2fc2a7733b6675a34354e15678f2b,c9f693515619455d
Lung_Opacity-3133.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3133.png,Lung_Opacity/masks/Lung_Opacity-3133.png,True,G05774,test,d5dba3f993952de0a1b0742bae075449,9c2a035f435d4f6c
Lung_Opacity-3134.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3134.png,Lung_Opacity/masks/Lung_Opacity-3134.png,True,G05775,train,9f9df5a20d13194aee0ca44f53c34a6d,c42d5b5a0c2cbdad
Lung_Opacity-3135.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3135.png,Lung_Opacity/masks/Lung_Opacity-3135.png,True,G05776,train,37b9c4707c518ebc16cf6e0e4003a373,9978971a58e978a5
Lung_Opacity-3136.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3136.png,Lung_Opacity/masks/Lung_Opacity-3136.png,True,G05777,val,d28c2195eb47f077c9a2e237990ce818,d83dc71b194f0e1c
Lung_Opacity-3137.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3137.png,Lung_Opacity/masks/Lung_Opacity-3137.png,True,G05778,train,11c408a9ccdc899475309211e57e3926,817a149e5a7b7133
Lung_Opacity-3138.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3138.png,Lung_Opacity/masks/Lung_Opacity-3138.png,True,G05779,test,2b379ef407253077cdde915e5f176b52,957f7416141f1d1c
Lung_Opacity-3139.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3139.png,Lung_Opacity/masks/Lung_Opacity-3139.png,True,G05780,train,88d2813882fe338fcf88f19f385772f7,917d085a385b5b5d
Lung_Opacity-314.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-314.png,Lung_Opacity/masks/Lung_Opacity-314.png,True,G05781,train,3ff675038cd8f64090e2daf3ab6fa9f8,c43e1743135c5b5b
Lung_Opacity-3140.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3140.png,Lung_Opacity/masks/Lung_Opacity-3140.png,True,G05782,train,cb64a8f26cd108ccb76c75a1d8f084da,917c8c195d79ce62
Lung_Opacity-3141.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3141.png,Lung_Opacity/masks/Lung_Opacity-3141.png,True,G05783,train,2609e20fb17f62068813d9187645edff,9578153a1f1e3c2d
Lung_Opacity-3142.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3142.png,Lung_Opacity/masks/Lung_Opacity-3142.png,True,G05784,train,b1f44859c22203bf3c604b3d088c552c,917b069a191d1f3b
Lung_Opacity-3143.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3143.png,Lung_Opacity/masks/Lung_Opacity-3143.png,True,G05785,train,6f04426fb373c69a7e10f39b58099d68,c07c0f4fe8713333
Lung_Opacity-3144.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3144.png,Lung_Opacity/masks/Lung_Opacity-3144.png,True,G05786,test,72685110c122510ce7066c25ee33e60c,973946362de5c8ca
Lung_Opacity-3145.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3145.png,Lung_Opacity/masks/Lung_Opacity-3145.png,True,G05787,train,60d182c5c4824217eb9f912e97948b9e,c42fd34b4e4a6f28
Lung_Opacity-3146.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3146.png,Lung_Opacity/masks/Lung_Opacity-3146.png,True,G05788,train,7d9a6474f3857e51c59590c53f21f031,d52a5a5f1ed8089d
Lung_Opacity-3147.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3147.png,Lung_Opacity/masks/Lung_Opacity-3147.png,True,G05789,train,66d7d8c6261e2bfb73a7d4d05553d5d7,d428915b4f2e3e1e
Lung_Opacity-3148.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3148.png,Lung_Opacity/masks/Lung_Opacity-3148.png,True,G05790,train,52f80be418e8d08a5b3d5fe2f53ba1ca,ed3c93431d5c4d1c
Lung_Opacity-3149.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3149.png,Lung_Opacity/masks/Lung_Opacity-3149.png,True,G05791,val,df4c57d7ecece71a67b6cedfdb5d7045,c83f06494cedacb5
Lung_Opacity-315.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-315.png,Lung_Opacity/masks/Lung_Opacity-315.png,True,G05792,train,68130e1e598fe1c4734592ab81b4117a,997d861a5deb1ac0
Lung_Opacity-3150.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3150.png,Lung_Opacity/masks/Lung_Opacity-3150.png,True,G05793,train,c30470cce2f5baf95dec6738aa69aafc,c02d5a4d0b3d2fb5
Lung_Opacity-3151.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3151.png,Lung_Opacity/masks/Lung_Opacity-3151.png,True,G05794,train,279e271c7a211da2b2af52253c502132,817d8418397d5f4e
Lung_Opacity-3152.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3152.png,Lung_Opacity/masks/Lung_Opacity-3152.png,True,G05795,test,374948e8363e05b73699c33a01c49f7b,c13b5686161a5bdb
Lung_Opacity-3153.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3153.png,Lung_Opacity/masks/Lung_Opacity-3153.png,True,G05796,val,b1f8e8f357b0020ef5d9986ea3db1766,c13d96ca4d6f0c39
Lung_Opacity-3154.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3154.png,Lung_Opacity/masks/Lung_Opacity-3154.png,True,G05797,val,6a6984daa60906458cc6ec2e21fbfde5,9178961a191f397d
Lung_Opacity-3155.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3155.png,Lung_Opacity/masks/Lung_Opacity-3155.png,True,G05798,test,f7de24be24f4ca42a1d6c0f8bb7258ee,d42f460e2c24f6e7
Lung_Opacity-3156.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3156.png,Lung_Opacity/masks/Lung_Opacity-3156.png,True,G05799,train,d2d2a5c6324ed36a3a1fcb2e3fe1bfc7,817e547f6f2864c4
Lung_Opacity-3157.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3157.png,Lung_Opacity/masks/Lung_Opacity-3157.png,True,G05800,train,c0c15823ffc00e676747e184bcc46577,d03f071e0d1d1d4f
Lung_Opacity-3158.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3158.png,Lung_Opacity/masks/Lung_Opacity-3158.png,True,G05801,train,2a0af5ff49e72ed045f22a7d1ce5ce42,d97d461a1c4b1d5a
Lung_Opacity-3159.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3159.png,Lung_Opacity/masks/Lung_Opacity-3159.png,True,G05802,train,019eae3c4b3d5859ff0385ed7eed2e15,d43e13121b5b1a7e
Lung_Opacity-316.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-316.png,Lung_Opacity/masks/Lung_Opacity-316.png,True,G05803,train,439774afb898361356fe30ba69c1ad18,817f06941a6b69ed
Lung_Opacity-3160.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3160.png,Lung_Opacity/masks/Lung_Opacity-3160.png,True,G05804,train,ee6a086366828705553ded393c29c950,c42f0d4d4c4eeae9
Lung_Opacity-3161.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3161.png,Lung_Opacity/masks/Lung_Opacity-3161.png,True,G05805,train,d870c52405a29c279e503c6be80e92d8,856a4f4f4d4e4c4d
Lung_Opacity-3162.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3162.png,Lung_Opacity/masks/Lung_Opacity-3162.png,True,G05806,train,8713105e9e3057b993f32593318d9adb,d03dd7421c5a5a5e
Lung_Opacity-3163.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3163.png,Lung_Opacity/masks/Lung_Opacity-3163.png,True,G05807,train,45b51553804bade8134f3114c543765f,c03d531b4a4f2d2f
Lung_Opacity-3164.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3164.png,Lung_Opacity/masks/Lung_Opacity-3164.png,True,G05808,train,fb7ae1d15ff8ef5064d4e2f8b044c8c7,983b57061e4d5f1a
Lung_Opacity-3165.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3165.png,Lung_Opacity/masks/Lung_Opacity-3165.png,True,G05809,val,b83a0831dce6c0f636bbba2f042631b6,817956521a5b4f79
Lung_Opacity-3166.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3166.png,Lung_Opacity/masks/Lung_Opacity-3166.png,True,G05810,val,6920c1bd604d9457869a599016c671be,d03dc3d24b438bab
Lung_Opacity-3167.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3167.png,Lung_Opacity/masks/Lung_Opacity-3167.png,True,G05811,val,41d0cda76bbfa2447d8f08eb2287c14b,c81d125b2939ebf1
Lung_Opacity-3168.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3168.png,Lung_Opacity/masks/Lung_Opacity-3168.png,True,G05812,test,7cb9cb6964f7129c81b666b13c40485f,d42cc3433d6ce8cb
Lung_Opacity-3169.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3169.png,Lung_Opacity/masks/Lung_Opacity-3169.png,True,G05813,test,9184fbc5cb3277ea83c333f319100650,81680f1f5f5e4a4e
Lung_Opacity-317.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-317.png,Lung_Opacity/masks/Lung_Opacity-317.png,True,G05814,test,63014c945bcdf18a8972e8bd1f018d74,c17bc6421a4e4f5b
Lung_Opacity-3170.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3170.png,Lung_Opacity/masks/Lung_Opacity-3170.png,True,G05815,train,503434834ce048978fe30a84a59b6e20,d959561e185b1b1e
Lung_Opacity-3171.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3171.png,Lung_Opacity/masks/Lung_Opacity-3171.png,True,G05816,train,570248e4ec53971f90847257a6d0b91e,956c06570f2d2d6d
Lung_Opacity-3172.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3172.png,Lung_Opacity/masks/Lung_Opacity-3172.png,True,G05817,train,e5319a10b227eeeaba2787d6824a9ddb,81695f8e3c452fc6
Lung_Opacity-3173.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3173.png,Lung_Opacity/masks/Lung_Opacity-3173.png,True,G05818,train,581eac2040829336ea49503007b7d918,972939da789664c6
Lung_Opacity-3174.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3174.png,Lung_Opacity/masks/Lung_Opacity-3174.png,True,G05819,train,c208072d2dd316270b3ca501a8ab02eb,cc35110f623f7c3a
Lung_Opacity-3175.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3175.png,Lung_Opacity/masks/Lung_Opacity-3175.png,True,G05820,train,75ebaffd9d53af71e010d0333599e26e,817b07141b5f5b1b
Lung_Opacity-3176.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3176.png,Lung_Opacity/masks/Lung_Opacity-3176.png,True,G05821,train,6a25ba6e2c6f9a76699299e7af3183bf,95632c7e66662696
Lung_Opacity-3177.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3177.png,Lung_Opacity/masks/Lung_Opacity-3177.png,True,G05822,train,4656793d26b77b67bd930967c92444a3,896c1c5b5d4d4d55
Lung_Opacity-3178.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3178.png,Lung_Opacity/masks/Lung_Opacity-3178.png,True,G05823,train,1f343054cd2b4409c18876c5e72697a4,c62f410f576e4a4d
Lung_Opacity-3179.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3179.png,Lung_Opacity/masks/Lung_Opacity-3179.png,True,G05824,train,f4eb1892df59b3a51a41c2844ebfb7f2,d03d1f140a3a3f3e
Lung_Opacity-318.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-318.png,Lung_Opacity/masks/Lung_Opacity-318.png,True,G05825,train,2359aad77cde8961af57d63dbecfa920,cd2d06970f0f1d1d
Lung_Opacity-3180.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3180.png,Lung_Opacity/masks/Lung_Opacity-3180.png,True,G05826,train,afd29dfefd6f341b8f2f022c06c40735,c93d065e186969ed
Lung_Opacity-3181.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3181.png,Lung_Opacity/masks/Lung_Opacity-3181.png,True,G05827,train,8c33930dd2a8677a13f33cca769447fc,d56e475a69e1e083
Lung_Opacity-3182.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3182.png,Lung_Opacity/masks/Lung_Opacity-3182.png,True,G05828,train,f006a42a71ae1d55d0e8e9bb00e72711,91794e16194f6d3c
Lung_Opacity-3183.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3183.png,Lung_Opacity/masks/Lung_Opacity-3183.png,True,G05829,train,95e653869922804db3860897b0a2254c,816e167a6fa7c885
Lung_Opacity-3184.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3184.png,Lung_Opacity/masks/Lung_Opacity-3184.png,True,G05830,train,ec5e2969d4ffd978e1d696bae0e388f6,d1191e1e1a39f9a5
Lung_Opacity-3185.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3185.png,Lung_Opacity/masks/Lung_Opacity-3185.png,True,G05831,val,ec9d69f6158dc85404491187f89b24ad,956c065e6b6d60ce
Lung_Opacity-3186.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3186.png,Lung_Opacity/masks/Lung_Opacity-3186.png,True,G05832,val,8efa54783f036ba9661a91efc01cccda,d03b431b1a3b393b
Lung_Opacity-3187.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3187.png,Lung_Opacity/masks/Lung_Opacity-3187.png,True,G05833,test,6af2e0255e4f422bde3298a31a9cc8a0,d03d5a4b483a6b1f
Lung_Opacity-3188.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3188.png,Lung_Opacity/masks/Lung_Opacity-3188.png,True,G05834,val,85adbe1e5cba5d855686ed01658b28b9,957c0e1a1d4d6c6d
Lung_Opacity-3189.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3189.png,Lung_Opacity/masks/Lung_Opacity-3189.png,True,G05835,test,2e306e74969ab9025b2cf01a7f483e1c,c02f434f4de7e5c0
Lung_Opacity-319.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-319.png,Lung_Opacity/masks/Lung_Opacity-319.png,True,G05836,test,afa88be1dd3a9df9adcd27b9a5cc0d18,df601f1a5b5b3818
Lung_Opacity-3190.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3190.png,Lung_Opacity/masks/Lung_Opacity-3190.png,True,G05837,train,1a806fa086472fbd0c198d296fa43fc2,94b863c766c13fc1
Lung_Opacity-3191.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3191.png,Lung_Opacity/masks/Lung_Opacity-3191.png,True,G05838,train,e92d0cbfd2b71493af0bab354072f9f7,c13f141a155b3b1f
Lung_Opacity-3192.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3192.png,Lung_Opacity/masks/Lung_Opacity-3192.png,True,G05839,train,fcdb4ebe90fa2edbb078e33646ab90be,817f004e1b591f5f
Lung_Opacity-3193.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3193.png,Lung_Opacity/masks/Lung_Opacity-3193.png,True,G05840,train,0a84f01df2e315d2ec5dcf2d98b84093,d03ec7571a0d2d1e
Lung_Opacity-3194.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3194.png,Lung_Opacity/masks/Lung_Opacity-3194.png,True,G05841,train,692ac2440d5b77bd381094160e6c8ac5,d43d528a0f0f1f0f
Lung_Opacity-3195.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3195.png,Lung_Opacity/masks/Lung_Opacity-3195.png,True,G05842,train,31de7494149c610f4d80f54c04a2a768,983b511a7abab2b2
Lung_Opacity-3196.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3196.png,Lung_Opacity/masks/Lung_Opacity-3196.png,True,G05843,train,21014b527b1112feb3ec04bc6863e384,91780e5e587b4bf0
Lung_Opacity-3197.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3197.png,Lung_Opacity/masks/Lung_Opacity-3197.png,True,G05844,train,f6227f1d81757268ebd92ab2e6e8d0f2,9578861e185d5d5e
Lung_Opacity-3198.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3198.png,Lung_Opacity/masks/Lung_Opacity-3198.png,True,G05845,train,38314e278ba3acbc190cd29b9edfd79d,952a47179e5f0e1c
Lung_Opacity-3199.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3199.png,Lung_Opacity/masks/Lung_Opacity-3199.png,True,G05846,train,d04f9d05a47632083887b797a12bc0b3,c77d3d1b0b489ca8
Lung_Opacity-32.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-32.png,Lung_Opacity/masks/Lung_Opacity-32.png,True,G05847,train,caa0f57f2fb34bc513d3bcb7a10a0c46,8c7f52431f1fd20c
Lung_Opacity-320.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-320.png,Lung_Opacity/masks/Lung_Opacity-320.png,True,G05848,train,0a28d9d503f40a96873380745cfea936,996bc6161a4b3d4d
Lung_Opacity-3200.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3200.png,Lung_Opacity/masks/Lung_Opacity-3200.png,True,G05849,train,8c4fbeef9843839e0fcf26d7ace7af87,c039133b1f3a3d2d
Lung_Opacity-3201.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3201.png,Lung_Opacity/masks/Lung_Opacity-3201.png,True,G05850,val,d325ac898dfdcf55c62fe1b44273403f,c129db1b5a5a5a5a
Lung_Opacity-3202.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3202.png,Lung_Opacity/masks/Lung_Opacity-3202.png,True,G05851,test,24d634705eb1cbf3aabbd036df4ad4f4,817b5718193f3e34
Lung_Opacity-3203.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3203.png,Lung_Opacity/masks/Lung_Opacity-3203.png,True,G05852,test,de812a390083db7a86821dd9563a2095,d43d61c54f0fc646
Lung_Opacity-3204.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3204.png,Lung_Opacity/masks/Lung_Opacity-3204.png,True,G05853,train,ecb16c2d842bf0861e74ea2d7274ca60,dd2dc21e1e4a3d68
Lung_Opacity-3205.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3205.png,Lung_Opacity/masks/Lung_Opacity-3205.png,True,G05854,train,913d1a0fe342246da06a1fe8fafe4010,dd0e065f4c7c48f8
Lung_Opacity-3206.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3206.png,Lung_Opacity/masks/Lung_Opacity-3206.png,True,G05855,train,564a8c75b0c9f5e38555fffd5c9481c4,d42d430f2e2c3e3d
Lung_Opacity-3207.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3207.png,Lung_Opacity/masks/Lung_Opacity-3207.png,True,G05856,val,90165fe9a2f4e4bc7552cd9cdcfcc87f,81395d1e78796666
Lung_Opacity-3208.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3208.png,Lung_Opacity/masks/Lung_Opacity-3208.png,True,G05857,train,bf06dd20cf9d76bf40c44c38cae13f7e,817b92483f1be935
Lung_Opacity-3209.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3209.png,Lung_Opacity/masks/Lung_Opacity-3209.png,True,G05858,train,b8c57609c5780b31ff67d4a3f3caa0ff,d87f465e4a1a1a4b
Lung_Opacity-321.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-321.png,Lung_Opacity/masks/Lung_Opacity-321.png,True,G05859,train,c72488432e7c47d137d0861bca2f5d30,c13b14961a595d5f
Lung_Opacity-3210.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3210.png,Lung_Opacity/masks/Lung_Opacity-3210.png,True,G05860,train,8e453bf15c2de45f2696eecd64341bef,946e07150e1d3e7e
Lung_Opacity-3211.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3211.png,Lung_Opacity/masks/Lung_Opacity-3211.png,True,G05861,val,a7ebb5bedb8876cab3abc2db7957e2af,8b781d1e11c9f9b1
Lung_Opacity-3212.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3212.png,Lung_Opacity/masks/Lung_Opacity-3212.png,True,G05862,train,2d691319da06eda5b931ae637e5851b3,817a101b5f5e3a1f
Lung_Opacity-3213.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3213.png,Lung_Opacity/masks/Lung_Opacity-3213.png,True,G05863,val,1251733ce0686510f3a3e10fa5498ee2,97681c5f6849c9ce
Lung_Opacity-3214.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3214.png,Lung_Opacity/masks/Lung_Opacity-3214.png,True,G05864,test,2f805cfd4806901a685ecf6ea4ad0d65,d42e43cd0c6ee378
Lung_Opacity-3215.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3215.png,Lung_Opacity/masks/Lung_Opacity-3215.png,True,G05865,train,72738e0a0e1ff17c2740bd87796715f9,d02b521d493b2d3f
Lung_Opacity-3216.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3216.png,Lung_Opacity/masks/Lung_Opacity-3216.png,True,G05866,train,6146de0bf61f252b2c06d4efeaf281dc,913dc64f105f4f49
Lung_Opacity-3217.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3217.png,Lung_Opacity/masks/Lung_Opacity-3217.png,True,G05867,train,60d6b7851385926ff463c72d2e577a99,dd2ac2531f49495d
Lung_Opacity-3218.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3218.png,Lung_Opacity/masks/Lung_Opacity-3218.png,True,G05868,train,fb323acd55d6101463e3e987aef108f2,aa3dc907b7959191
Lung_Opacity-3219.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3219.png,Lung_Opacity/masks/Lung_Opacity-3219.png,True,G05869,train,29844a91a9e55cd62284fcafd7adac74,8069c7165e471f1f
Lung_Opacity-322.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-322.png,Lung_Opacity/masks/Lung_Opacity-322.png,True,G05870,train,6f0a43e0ffc70c20759d0ccb05f41b77,d42bf6d217c2321e
Lung_Opacity-3220.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3220.png,Lung_Opacity/masks/Lung_Opacity-3220.png,True,G05871,train,7f9ec6b75da356bb150b632502a58046,81789e1bfb4da017
Lung_Opacity-3221.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3221.png,Lung_Opacity/masks/Lung_Opacity-3221.png,True,G05872,val,ad802be46bf262cc851bdb54964e1373,892e165a6d7a7163
Lung_Opacity-3222.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3222.png,Lung_Opacity/masks/Lung_Opacity-3222.png,True,G05873,val,1fd7a9d49f4bd1592243c3e02fb834bd,813aff04fe501a5e
Lung_Opacity-3223.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3223.png,Lung_Opacity/masks/Lung_Opacity-3223.png,True,G05874,test,300a92927b8e21b08359b1bd1f2116ee,d52b56061e1d555d
Lung_Opacity-3224.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3224.png,Lung_Opacity/masks/Lung_Opacity-3224.png,True,G05875,test,ead5a11a0422a16d86f362c252f25b52,c02f97490d5d5c5e
Lung_Opacity-3225.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3225.png,Lung_Opacity/masks/Lung_Opacity-3225.png,True,G05876,test,65c80e29ab0273650c24d29a26b74868,d03d420a0f6f2c7f
Lung_Opacity-3226.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3226.png,Lung_Opacity/masks/Lung_Opacity-3226.png,True,G05877,train,cca4a4800c860e3dc46d0182fc1b641f,816e477ae2e4c4d3
Lung_Opacity-3227.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3227.png,Lung_Opacity/masks/Lung_Opacity-3227.png,True,G05878,train,0d23d56075fcfd88c135952e75fc803d,9370141e123b7b3f
Lung_Opacity-3228.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3228.png,Lung_Opacity/masks/Lung_Opacity-3228.png,True,G05879,train,23ca3ad134515c396a9d9e4d83c33245,dc46434f0e4f2e2b
Lung_Opacity-3229.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3229.png,Lung_Opacity/masks/Lung_Opacity-3229.png,True,G05880,train,3bcf4f7952f4b161a37733c49efc2e26,817d141e1c5e7a7a
Lung_Opacity-323.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-323.png,Lung_Opacity/masks/Lung_Opacity-323.png,True,G05881,train,e1697e0ddaf2d44b344ba0c53e589915,d43e07474f5e2c4c
Lung_Opacity-3230.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3230.png,Lung_Opacity/masks/Lung_Opacity-3230.png,True,G05882,train,a9a3bf5b8c1f83d7944305e3006f242b,917bc61a594b595a
Lung_Opacity-3231.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3231.png,Lung_Opacity/masks/Lung_Opacity-3231.png,True,G05883,train,c94ffff6d995e7c7425c892e52d8811d,8178005f5f5c5f5c
Lung_Opacity-3232.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3232.png,Lung_Opacity/masks/Lung_Opacity-3232.png,True,G05884,train,feb73c8f55f4967037ac64f8c5016a60,907d4a1e7d2931b3
Lung_Opacity-3233.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3233.png,Lung_Opacity/masks/Lung_Opacity-3233.png,True,G05885,train,a62412ff7e0b3c46c6be8aab064a908a,d13b4fc652191a4f
Lung_Opacity-3234.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3234.png,Lung_Opacity/masks/Lung_Opacity-3234.png,True,G05886,train,09f5c220b07ef39f75e20ce440a0f078,c43e0f0a4d6c7e6a
Lung_Opacity-3235.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3235.png,Lung_Opacity/masks/Lung_Opacity-3235.png,True,G05887,val,da63237ed23dcad3899ad3298a2270e4,81780f5e1a5b497d
Lung_Opacity-3236.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3236.png,Lung_Opacity/masks/Lung_Opacity-3236.png,True,G05888,train,27e6c5a7df4a66db1c692b0513f6ee1d,847e0f4a2a7e6c6c
Lung_Opacity-3237.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3237.png,Lung_Opacity/masks/Lung_Opacity-3237.png,True,G05889,train,66a0094cb4f690243f2e2ee250992f3a,bb7b861a1bb980ac
Lung_Opacity-3238.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3238.png,Lung_Opacity/masks/Lung_Opacity-3238.png,True,G05890,train,7e603880343baff5a9d95c370bc74769,9968065a6b776726
Lung_Opacity-3239.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3239.png,Lung_Opacity/masks/Lung_Opacity-3239.png,True,G05891,val,ef2b1a708a61643b4016222a2f77991a,916a061e1f0f5f1d
Lung_Opacity-324.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-324.png,Lung_Opacity/masks/Lung_Opacity-324.png,True,G05892,test,ca705760af0c98347a3143fe88b36b7e,837a051417527f7b
Lung_Opacity-3240.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3240.png,Lung_Opacity/masks/Lung_Opacity-3240.png,True,G05893,train,e5dfe6284ed7026837c16504d3aaa032,fb72f6f030233138
Lung_Opacity-3241.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3241.png,Lung_Opacity/masks/Lung_Opacity-3241.png,True,G05894,test,94bbdad16311a1096e8f39edbc28bef8,9528071a1b1f5b5f
Lung_Opacity-3242.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3242.png,Lung_Opacity/masks/Lung_Opacity-3242.png,True,G05895,train,bb576afdbd1371104333608b582fec0a,9d4f475e5843435a
Lung_Opacity-3243.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3243.png,Lung_Opacity/masks/Lung_Opacity-3243.png,True,G05896,val,c5552d191c6f11b7b9217751b6e9ea5e,c42f1b4d5e6e0e0b
Lung_Opacity-3244.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3244.png,Lung_Opacity/masks/Lung_Opacity-3244.png,True,G05897,val,76cac96adfec73afbcacb2d5333906db,897d045e197969e9
Lung_Opacity-3245.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3245.png,Lung_Opacity/masks/Lung_Opacity-3245.png,True,G05898,train,5adcc6185399721164afd25e0d8e1f65,dd38d2c30b0f86ab
Lung_Opacity-3246.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3246.png,Lung_Opacity/masks/Lung_Opacity-3246.png,True,G05899,val,bcd17391bc7c228e8907ade80184cf55,9d6c0a792b32e372
Lung_Opacity-3247.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3247.png,Lung_Opacity/masks/Lung_Opacity-3247.png,True,G05900,test,4529f8e3e5eb5fd83edfcbba493498fa,917be685bd12c607
Lung_Opacity-3248.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3248.png,Lung_Opacity/masks/Lung_Opacity-3248.png,True,G05901,test,994e411fb66253c60bfc45c6496c9d09,95686e9e6ab162a5
Lung_Opacity-3249.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3249.png,Lung_Opacity/masks/Lung_Opacity-3249.png,True,G05902,train,b24e2a970a950e1c2d665fde48f7d002,dd2992130f0f1b1f
Lung_Opacity-325.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-325.png,Lung_Opacity/masks/Lung_Opacity-325.png,True,G05903,train,6f91b50db97298e4a8d3b519dc69eccb,9c66430b4c6e2eee
Lung_Opacity-3250.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3250.png,Lung_Opacity/masks/Lung_Opacity-3250.png,True,G05904,train,878e7916d27419f24bd9eb7df26e5bc6,c43d0b0a4a4f4f7e
Lung_Opacity-3251.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3251.png,Lung_Opacity/masks/Lung_Opacity-3251.png,True,G05905,train,009aa51460e177858e653b1905300f41,cd2e504b1d6a4b3e
Lung_Opacity-3252.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3252.png,Lung_Opacity/masks/Lung_Opacity-3252.png,True,G05906,val,0b4ce1b0ad858be963b9caafeba23d6e,d0295b5f4b0d4b2d
Lung_Opacity-3253.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3253.png,Lung_Opacity/masks/Lung_Opacity-3253.png,True,G05907,train,3ee07d42f04e1cf7547e262bb5bb8875,8978175279f9e988
Lung_Opacity-3254.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3254.png,Lung_Opacity/masks/Lung_Opacity-3254.png,True,G05908,train,16d3b1ba1313c63361db625812db04bd,807f8f161a5b3b58
Lung_Opacity-3255.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3255.png,Lung_Opacity/masks/Lung_Opacity-3255.png,True,G05909,train,38753585dfbc0d391a889a4b7e04b688,887f061e5a7d585c
Lung_Opacity-3256.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3256.png,Lung_Opacity/masks/Lung_Opacity-3256.png,True,G05910,train,febd3b81e2fa0ca4dc2ac64dd6955bec,817f80ee0858c77f
Lung_Opacity-3257.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3257.png,Lung_Opacity/masks/Lung_Opacity-3257.png,True,G05911,train,8698da347f1ec9d847206ebbc788957a,887d861ec09fd2c7
Lung_Opacity-3258.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3258.png,Lung_Opacity/masks/Lung_Opacity-3258.png,True,G05912,train,3944a81288749068d4cef88ed9a67bb9,d12d06871e1f0d5f
Lung_Opacity-3259.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3259.png,Lung_Opacity/masks/Lung_Opacity-3259.png,True,G05913,test,2952c7553643624433bf450e5169eff9,dc2d4a420f0e6db7
Lung_Opacity-326.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-326.png,Lung_Opacity/masks/Lung_Opacity-326.png,True,G05914,train,4d9701e075fc8428cdfb92b98e17ddb2,956a181e1b1a5b5f
Lung_Opacity-3260.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3260.png,Lung_Opacity/masks/Lung_Opacity-3260.png,True,G05915,train,f967f91982b34dcd304021c7a298885b,91781c1c39e9ecdc
Lung_Opacity-3261.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3261.png,Lung_Opacity/masks/Lung_Opacity-3261.png,True,G05916,test,b701cb180fe73a22447c6b7bf622b08a,d0653e0f2d2f6c68
Lung_Opacity-3262.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3262.png,Lung_Opacity/masks/Lung_Opacity-3262.png,True,G05917,train,45a41339593590fcb0459fa94c8b1481,c07f5f0d5819192f
Lung_Opacity-3263.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3263.png,Lung_Opacity/masks/Lung_Opacity-3263.png,True,G05918,train,cdbe68ff46e3d0b5d17f42a7a621d783,d93d40c34b1f1b4b
Lung_Opacity-3264.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3264.png,Lung_Opacity/masks/Lung_Opacity-3264.png,True,G05919,train,042ef89721530a523533f56a01a8bfff,817af492121379bf
Lung_Opacity-3265.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3265.png,Lung_Opacity/masks/Lung_Opacity-3265.png,True,G05920,train,eb958ea4834873247e0f1d6cb1b7d9e5,856e474f4e4d5c49
Lung_Opacity-3266.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3266.png,Lung_Opacity/masks/Lung_Opacity-3266.png,True,G05921,train,58c1c6e5570e612d5cc453f752ad947b,d13b569038393dbc
Lung_Opacity-3267.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3267.png,Lung_Opacity/masks/Lung_Opacity-3267.png,True,G05922,train,ada27c1309a89b7c1109af8a8de364c3,d53dc6021a1a5f4f
Lung_Opacity-3268.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3268.png,Lung_Opacity/masks/Lung_Opacity-3268.png,True,G05923,train,694ead373a866e0429b3f5223e664b92,917b961a3a3d3a2c
Lung_Opacity-3269.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3269.png,Lung_Opacity/masks/Lung_Opacity-3269.png,True,G05924,train,6ba2118be133b10d9861d3db05e849bd,d42b431d4c3b1e3b
Lung_Opacity-327.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-327.png,Lung_Opacity/masks/Lung_Opacity-327.png,True,G05925,train,2cc29b28977dd17ab530da82e94450f1,947f0247161f1e5d
Lung_Opacity-3270.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3270.png,Lung_Opacity/masks/Lung_Opacity-3270.png,True,G05926,train,181d383c64b200cfab17153cc763741b,c06d2fc63d823d9a
Lung_Opacity-3271.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3271.png,Lung_Opacity/masks/Lung_Opacity-3271.png,True,G05927,train,a90cc35338189f0f66b945a48aa84983,c938145a1b59797b
Lung_Opacity-3272.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3272.png,Lung_Opacity/masks/Lung_Opacity-3272.png,True,G05928,train,148807040f7e78f36181c70b3f2afb1a,c42f464f4cec68f1
Lung_Opacity-3273.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3273.png,Lung_Opacity/masks/Lung_Opacity-3273.png,True,G05929,train,4439b20b7ed74ac8858b75354b9782f1,887f961213171f5d
Lung_Opacity-3274.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3274.png,Lung_Opacity/masks/Lung_Opacity-3274.png,True,G05930,train,15dc812e293f91312fef5cf99ba6b07f,910f0dcfe1635734
Lung_Opacity-3275.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3275.png,Lung_Opacity/masks/Lung_Opacity-3275.png,True,G05931,val,8a36e240a33f19c87cfd090b247866e0,dc2d450d4a0f2e7e
Lung_Opacity-3276.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3276.png,Lung_Opacity/masks/Lung_Opacity-3276.png,True,G05932,val,a26ea7167ecacffc87e64da50332fe83,896d06160f5f5c5d
Lung_Opacity-3277.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3277.png,Lung_Opacity/masks/Lung_Opacity-3277.png,True,G05933,val,1929e0edd7f51f9ebb30e613e0ab66d9,d42f4f082c2c3fba
Lung_Opacity-3278.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3278.png,Lung_Opacity/masks/Lung_Opacity-3278.png,True,G05934,test,8325d511a9129a29c5d335c2931d84d5,9c6a033d423e6677
Lung_Opacity-3279.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3279.png,Lung_Opacity/masks/Lung_Opacity-3279.png,True,G05935,test,691711e69e3d90d29a526d6c3dd59cd5,dc2c430f1f4b0f0f
Lung_Opacity-328.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-328.png,Lung_Opacity/masks/Lung_Opacity-328.png,True,G05936,test,83e7f32e38d79dd0a7563983ae77ecf8,d46f8713834a8e4f
Lung_Opacity-3280.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3280.png,Lung_Opacity/masks/Lung_Opacity-3280.png,True,G05937,train,06e208eef7062fdcbd849f68e377d3e0,8d68975a0f5c382f
Lung_Opacity-3281.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3281.png,Lung_Opacity/masks/Lung_Opacity-3281.png,True,G05938,val,e8cc87a6c67a511d87e383170a7f7553,c93d571a1c4d4e1e
Lung_Opacity-3282.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3282.png,Lung_Opacity/masks/Lung_Opacity-3282.png,True,G05939,train,393933ed3c776709dc27a145f9783e7e,823d715a7a38794b
Lung_Opacity-3283.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3283.png,Lung_Opacity/masks/Lung_Opacity-3283.png,True,G05940,train,8e6e3b2dc970a7f5c119c49d5aa9010f,9ba9c6585b595949
Lung_Opacity-3284.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3284.png,Lung_Opacity/masks/Lung_Opacity-3284.png,True,G05941,train,ff45b640711863cbafee097f93857976,802fc7574f4f0a1d
Lung_Opacity-3285.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3285.png,Lung_Opacity/masks/Lung_Opacity-3285.png,True,G05942,train,2bdfcee42a5ddf4f3977979b152a9c7a,c83d1647075e5b1d
Lung_Opacity-3286.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3286.png,Lung_Opacity/masks/Lung_Opacity-3286.png,True,G05943,train,f1088e1282e9a52040a228d6354ec2d1,817b048e1f785bcd
Lung_Opacity-3287.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3287.png,Lung_Opacity/masks/Lung_Opacity-3287.png,True,G05944,train,2f01ab40f281d594eb9ca15072fa3dd8,817b17061828ffeb
Lung_Opacity-3288.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3288.png,Lung_Opacity/masks/Lung_Opacity-3288.png,True,G05945,train,d4706785a27ea7d3a6c4afec45a1f7c5,9d6f861b094d0d4f
Lung_Opacity-3289.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3289.png,Lung_Opacity/masks/Lung_Opacity-3289.png,True,G05946,train,c1aa767137c1c52702dceb99a7e4e146,917fec004d073b1f
Lung_Opacity-329.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-329.png,Lung_Opacity/masks/Lung_Opacity-329.png,True,G05947,train,c0a8e99c1c793be358d5cfca26a9b078,81781e861859de7f
Lung_Opacity-3290.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3290.png,Lung_Opacity/masks/Lung_Opacity-3290.png,True,G05948,train,5fda4fae5aa2f7de2e036673ffa4efaf,8f2c6b31e39a8ccc
Lung_Opacity-3291.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3291.png,Lung_Opacity/masks/Lung_Opacity-3291.png,True,G05949,train,1fd1e0ddb5d6caeedb566176e8999992,896887163bb51e5b
Lung_Opacity-3292.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3292.png,Lung_Opacity/masks/Lung_Opacity-3292.png,True,G05950,train,3ea22b9d47a54c076ee45299a82b5b8a,917b581a7b79e8a0
Lung_Opacity-3293.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3293.png,Lung_Opacity/masks/Lung_Opacity-3293.png,True,G05951,test,fdc77ee66938eafc28ca447352b0e13e,ca25db998d889697
Lung_Opacity-3294.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3294.png,Lung_Opacity/masks/Lung_Opacity-3294.png,True,G05952,train,899cc23e48c8568304cd29c76d1df7d0,897846005b79f97d
Lung_Opacity-3295.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3295.png,Lung_Opacity/masks/Lung_Opacity-3295.png,True,G05953,val,bc41055b478be9b2d801bb866a01a395,947ea75b81170e87
Lung_Opacity-3296.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3296.png,Lung_Opacity/masks/Lung_Opacity-3296.png,True,G05954,val,04ab99a65aeb8ef89a5b3100f439adb3,c52e521f1f425b1e
Lung_Opacity-3297.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3297.png,Lung_Opacity/masks/Lung_Opacity-3297.png,True,G05955,test,94bf6f7351b6d5ffab4b386e58d51542,c46e014a2e6e7ef4
Lung_Opacity-3298.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3298.png,Lung_Opacity/masks/Lung_Opacity-3298.png,True,G05956,test,899c2010a691141a55dab77acd68770a,c26c2f6f69606571
Lung_Opacity-3299.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3299.png,Lung_Opacity/masks/Lung_Opacity-3299.png,True,G05957,train,64dda48aadac8bceaca85b70f8d65d81,d01d121b1b3b79ba
Lung_Opacity-33.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-33.png,Lung_Opacity/masks/Lung_Opacity-33.png,True,G05958,train,ca2472b567c9a20bd352dded3ec65b3d,987b460e49496ded
Lung_Opacity-330.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-330.png,Lung_Opacity/masks/Lung_Opacity-330.png,True,G05959,train,fc7093fec79cb26bf61ed45d7d1b18f8,997817861e89b8be
Lung_Opacity-3300.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3300.png,Lung_Opacity/masks/Lung_Opacity-3300.png,True,G05960,train,918001c32060e0e29412b5cf0f4b8d9a,817b5ec61a195a1f
Lung_Opacity-3301.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3301.png,Lung_Opacity/masks/Lung_Opacity-3301.png,True,G05961,train,400976a5c9298fcbc81b7235635f7194,996c16581d5879e7
Lung_Opacity-3302.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3302.png,Lung_Opacity/masks/Lung_Opacity-3302.png,True,G05962,train,3e87547b741b41b6e47601e7c30f065f,817807061e7e3d7d
Lung_Opacity-3303.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3303.png,Lung_Opacity/masks/Lung_Opacity-3303.png,True,G05963,train,f55f0c95070b27e80426af88684f4d56,87680c1f393a787b
Lung_Opacity-3304.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3304.png,Lung_Opacity/masks/Lung_Opacity-3304.png,True,G05964,train,75c5a5544f6eba0ac929eb8bd6206d15,c90c961a59784b7f
Lung_Opacity-3305.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3305.png,Lung_Opacity/masks/Lung_Opacity-3305.png,True,G05965,train,25b4b4d56109358d6c253dbd6d8e3027,c93e38cd3ac333c1
Lung_Opacity-3306.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3306.png,Lung_Opacity/masks/Lung_Opacity-3306.png,True,G05966,train,1b8abfd034cd9b059439a67a02e4d79a,894f464b4971f6b4
Lung_Opacity-3307.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3307.png,Lung_Opacity/masks/Lung_Opacity-3307.png,True,G05967,train,2e1d8d3b6298d1aecbdca67886fcad91,9978161e4b1f1d1d
Lung_Opacity-3308.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3308.png,Lung_Opacity/masks/Lung_Opacity-3308.png,True,G05968,val,3bea55e68f8b0f653cfdfc3c55ec7239,87780c1e1e1d1e7b
Lung_Opacity-3309.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3309.png,Lung_Opacity/masks/Lung_Opacity-3309.png,True,G05969,train,ed22bcafac899345c47640993b343371,817b561a182d3bbd
Lung_Opacity-331.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-331.png,Lung_Opacity/masks/Lung_Opacity-331.png,True,G05970,train,85dacd8e4d360859d84ca1174113ea11,807f87163c795959
Lung_Opacity-3310.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3310.png,Lung_Opacity/masks/Lung_Opacity-3310.png,True,G05971,val,88d6293de2815e6f3de036da90a91498,cc75ba49cb2a546c
Lung_Opacity-3311.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3311.png,Lung_Opacity/masks/Lung_Opacity-3311.png,True,G05972,test,0fac220307a4169209a9647b2395c6ca,953b520a1a393d3f
Lung_Opacity-3312.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3312.png,Lung_Opacity/masks/Lung_Opacity-3312.png,True,G05973,test,7bb4dab2baaacba4849dc29e25fa0a24,d13b5f021e5b1b0b
Lung_Opacity-3313.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3313.png,Lung_Opacity/masks/Lung_Opacity-3313.png,True,G05974,test,833cb642bc97f12e3739b4b1fad358f3,cc2a5723d72b552a
Lung_Opacity-3314.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3314.png,Lung_Opacity/masks/Lung_Opacity-3314.png,True,G05975,train,b70ab455d27fdfb285e8c9ee4664047a,dc3dd34b0f0b1a1a
Lung_Opacity-3315.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3315.png,Lung_Opacity/masks/Lung_Opacity-3315.png,True,G05976,train,18e82eccdc152c24c68ad88e9dcfe3ef,962a510f8f5e0f2e
Lung_Opacity-3316.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3316.png,Lung_Opacity/masks/Lung_Opacity-3316.png,True,G05977,val,b160df9ea3b9d353af98243a5417e678,d03d4b471d1c6d2d
Lung_Opacity-3317.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3317.png,Lung_Opacity/masks/Lung_Opacity-3317.png,True,G05978,val,969e9717b1d8c20e085c80c92cb7228e,816db61a0b5d2d6d
Lung_Opacity-3318.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3318.png,Lung_Opacity/masks/Lung_Opacity-3318.png,True,G05979,test,cc8b0c164c1b82f363f1d8fbdac188c4,d07dc6421b4f293b
Lung_Opacity-3319.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3319.png,Lung_Opacity/masks/Lung_Opacity-3319.png,True,G05980,train,c9fba77d7e0e9059bf68f73d2824a081,83701c173b5a7a73
Lung_Opacity-332.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-332.png,Lung_Opacity/masks/Lung_Opacity-332.png,True,G05981,train,0c5d46accdc0dedfb179898ef5a8d506,987bc7161e437859
Lung_Opacity-3320.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3320.png,Lung_Opacity/masks/Lung_Opacity-3320.png,True,G05982,val,00f566e64046f0cc1576f382358066f6,d039531b4a5f1b2b
Lung_Opacity-3321.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3321.png,Lung_Opacity/masks/Lung_Opacity-3321.png,True,G05983,test,233dc28b3bc34ba5cca8f21c2da85506,c6742f0f69687871
Lung_Opacity-3322.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3322.png,Lung_Opacity/masks/Lung_Opacity-3322.png,True,G05984,train,ec0ad48e89dac303f9c55ebeb67582f5,817a54061a5b5f5f
Lung_Opacity-3323.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3323.png,Lung_Opacity/masks/Lung_Opacity-3323.png,True,G05985,train,da24d5570c5aa7790a948de1fe882287,992ac75f78691782
Lung_Opacity-3324.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3324.png,Lung_Opacity/masks/Lung_Opacity-3324.png,True,G05986,train,6a42c5aa3cfdfcf3a0a2117625d7b395,80546e8f7d8c7cd4
Lung_Opacity-3325.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3325.png,Lung_Opacity/masks/Lung_Opacity-3325.png,True,G05987,train,40049e2dfaa7594af2fde92d1186fe2c,d46f0b7d30703333
Lung_Opacity-3326.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3326.png,Lung_Opacity/masks/Lung_Opacity-3326.png,True,G05988,train,bb8b0bc6d4f0eff952f75a38ad862f25,dc3dc95d0a4e0f4a
Lung_Opacity-3327.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3327.png,Lung_Opacity/masks/Lung_Opacity-3327.png,True,G05989,train,ebfd7b0be68f4540dce3138c2ccc798a,817e074a99cbe596
Lung_Opacity-3328.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3328.png,Lung_Opacity/masks/Lung_Opacity-3328.png,True,G05990,train,76361bea7ea07ab6e447c8960716fba9,81786f5f1e5a581a
Lung_Opacity-3329.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3329.png,Lung_Opacity/masks/Lung_Opacity-3329.png,True,G05991,train,82ac57c5548a6ee25409a9ae1ee8196e,d83f5218183e3eb6
Lung_Opacity-333.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-333.png,Lung_Opacity/masks/Lung_Opacity-333.png,True,G05992,train,1e890df38cf2c9bb9e5d3f59d087e16e,dd2ac75f055c5c0c
Lung_Opacity-3330.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3330.png,Lung_Opacity/masks/Lung_Opacity-3330.png,True,G05993,train,bf6ab5101d9397781da163566ff5a5d7,917aa7508b533b53
Lung_Opacity-3331.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3331.png,Lung_Opacity/masks/Lung_Opacity-3331.png,True,G05994,train,6adbf0802c06be3f750c8bdd9a523678,dc2d42130f6c3cbd
Lung_Opacity-3332.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3332.png,Lung_Opacity/masks/Lung_Opacity-3332.png,True,G05995,val,b5d159ca78b32340f508ba9b353f7328,902fc61a4c5f5d4e
Lung_Opacity-3333.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3333.png,Lung_Opacity/masks/Lung_Opacity-3333.png,True,G05996,test,ddb53bd22311b610961b894d3411520a,d93b05161c787bb1
Lung_Opacity-3334.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3334.png,Lung_Opacity/masks/Lung_Opacity-3334.png,True,G05997,train,7aa8e91492d46bfa1eccff40e4699505,8b6886161e1f1f1f
Lung_Opacity-3335.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3335.png,Lung_Opacity/masks/Lung_Opacity-3335.png,True,G05998,train,c15468b562f468d0c8099d36f1a97892,81689616595d7b79
Lung_Opacity-3336.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3336.png,Lung_Opacity/masks/Lung_Opacity-3336.png,True,G05999,train,8671fb277b92726a61e60c652f33e366,867d62a5dc0f7321
Lung_Opacity-3337.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3337.png,Lung_Opacity/masks/Lung_Opacity-3337.png,True,G06000,train,cbaaf79b60883f2b51f991fd63d431e4,d42f424b4f4e0f0f
Lung_Opacity-3338.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3338.png,Lung_Opacity/masks/Lung_Opacity-3338.png,True,G06001,train,12bea7bd25bd7447093448ef94a55324,d03c6dc22bd33bc1
Lung_Opacity-3339.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3339.png,Lung_Opacity/masks/Lung_Opacity-3339.png,True,G06002,train,57ccc854090658211914a79dfe39e707,dc3d5b174e181a1e
Lung_Opacity-334.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-334.png,Lung_Opacity/masks/Lung_Opacity-334.png,True,G06003,train,1f870ae88999ffe36655ef1c21da2b3c,947e071c493f4b1b
Lung_Opacity-3340.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3340.png,Lung_Opacity/masks/Lung_Opacity-3340.png,True,G06004,val,2da27264a68ea95ed80579d13f2857e5,d02f470a1d4b1f1f
Lung_Opacity-3341.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3341.png,Lung_Opacity/masks/Lung_Opacity-3341.png,True,G06005,train,a4b7024ce3fd6967a349885943218361,807f56141b1f5717
Lung_Opacity-3342.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3342.png,Lung_Opacity/masks/Lung_Opacity-3342.png,True,G06006,test,0ed227e2b925af47248d2399f444e461,817e0617487f7f30
Lung_Opacity-3343.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3343.png,Lung_Opacity/masks/Lung_Opacity-3343.png,True,G06007,train,57928ee18b0d649fc75f8a9a1446e42f,883d04c7165afd57
Lung_Opacity-3344.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3344.png,Lung_Opacity/masks/Lung_Opacity-3344.png,True,G06008,val,7d0403555c75495ed627fa356899ba58,9128065f5d7b5b51
Lung_Opacity-3345.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3345.png,Lung_Opacity/masks/Lung_Opacity-3345.png,True,G06009,test,57c702e0ff1352d25d4f6f0d15b0979f,c02d9a871d1e5d5d
Lung_Opacity-3346.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3346.png,Lung_Opacity/masks/Lung_Opacity-3346.png,True,G06010,train,640e1de2875372e66bec12e8bf0d0bbf,956c2e4a4d7c686d
Lung_Opacity-3347.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3347.png,Lung_Opacity/masks/Lung_Opacity-3347.png,True,G06011,test,9e2b0de7dfa3b888166e77bef2badf57,c03152526f7b3e2d
Lung_Opacity-3348.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3348.png,Lung_Opacity/masks/Lung_Opacity-3348.png,True,G06012,train,bd389b761613cbcb9d91a491267a0f9d,817e101e7b1b3b3a
Lung_Opacity-3349.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3349.png,Lung_Opacity/masks/Lung_Opacity-3349.png,True,G06013,train,99ee56b86c7958d56867e733a61d57b0,c03f7b5b5a5d6808
Lung_Opacity-335.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-335.png,Lung_Opacity/masks/Lung_Opacity-335.png,True,G06014,train,c049f7aaee2176a1a03b7408d708a264,81aa800a1f5f797f
Lung_Opacity-3350.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3350.png,Lung_Opacity/masks/Lung_Opacity-3350.png,True,G06015,train,360d8557264db61503f0d98dd595e54a,d42f435b0a0a3f2f
Lung_Opacity-3351.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3351.png,Lung_Opacity/masks/Lung_Opacity-3351.png,True,G06016,val,ee318d5f5f309a97b3ee0551861d51a8,d52e45434b4f2e4e
Lung_Opacity-3352.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3352.png,Lung_Opacity/masks/Lung_Opacity-3352.png,True,G06017,train,36d88132830eebafc1d895ad4aed6ba0,d96c061a153b0f7b
Lung_Opacity-3353.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3353.png,Lung_Opacity/masks/Lung_Opacity-3353.png,True,G06018,train,69567af9f43f3ba45c2892c36c919f31,957c06433b9f68a5
Lung_Opacity-3354.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3354.png,Lung_Opacity/masks/Lung_Opacity-3354.png,True,G06019,train,d82f2bd1251815d87ba0656fda193449,c83d564b493c3c2f
Lung_Opacity-3355.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3355.png,Lung_Opacity/masks/Lung_Opacity-3355.png,True,G06020,train,9c8833b2ecf6790d7b0ef8c7b4ec2daa,c627430f2667e5f0
Lung_Opacity-3356.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3356.png,Lung_Opacity/masks/Lung_Opacity-3356.png,True,G06021,train,62a4332361ee70e0178da1b7a48e9288,99780a1a1b595bfa
Lung_Opacity-3357.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3357.png,Lung_Opacity/masks/Lung_Opacity-3357.png,True,G06022,train,917b9d91fc47f52c1314185c421e6617,95681f5e1a59395c
Lung_Opacity-3358.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3358.png,Lung_Opacity/masks/Lung_Opacity-3358.png,True,G06023,train,28fba30adefe9b51ce91cfa4e25ef8a9,d56a4555486b2b2f
Lung_Opacity-3359.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3359.png,Lung_Opacity/masks/Lung_Opacity-3359.png,True,G06024,train,56c7bd885220c090d86c3701b18e95f7,dd2ed0034f4c0f4f
Lung_Opacity-336.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-336.png,Lung_Opacity/masks/Lung_Opacity-336.png,True,G06025,train,45a0974d21acb6081397b047106ff688,d4495b0a0c2f3d3f
Lung_Opacity-3360.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3360.png,Lung_Opacity/masks/Lung_Opacity-3360.png,True,G06026,train,fed9402067452a38738f56c4afe46c07,882d128b0f2f1f3f
Lung_Opacity-3361.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3361.png,Lung_Opacity/masks/Lung_Opacity-3361.png,True,G06027,train,13c8b1b9901324e9a4309d695adda98a,dc6f03092d3c2eba
Lung_Opacity-3362.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3362.png,Lung_Opacity/masks/Lung_Opacity-3362.png,True,G06028,train,07303a29e5d2a088809a986dd4d2f297,c43d4a086e6c6b6f
Lung_Opacity-3363.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3363.png,Lung_Opacity/masks/Lung_Opacity-3363.png,True,G06029,train,cfd2d2cf309228cddb40a093811dad9e,817896161d5e5e5d
Lung_Opacity-3364.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3364.png,Lung_Opacity/masks/Lung_Opacity-3364.png,True,G06030,train,1f65d5024336a4eb37c0659211160198,816f3e70616f2e4a
Lung_Opacity-3365.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3365.png,Lung_Opacity/masks/Lung_Opacity-3365.png,True,G06031,val,e317dd33a84b80885269b7055b07a9b2,c83d590a191f1f1f
Lung_Opacity-3366.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3366.png,Lung_Opacity/masks/Lung_Opacity-3366.png,True,G06032,train,846760148a5b78103c856216e551498f,813fec8e9494967a
Lung_Opacity-3367.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3367.png,Lung_Opacity/masks/Lung_Opacity-3367.png,True,G06033,val,797bba2a6a637dd90eb98aa3ae1cfc8d,837b8c1d361f0b1b
Lung_Opacity-3368.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3368.png,Lung_Opacity/masks/Lung_Opacity-3368.png,True,G06034,train,8b825c5ce2704f076c6f0ae00cc5b4d3,d83d421f4b0d3d1d
Lung_Opacity-3369.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3369.png,Lung_Opacity/masks/Lung_Opacity-3369.png,True,G06035,val,f6ce34a30d629e913c820b74496a0c49,d819530b3d392d3d
Lung_Opacity-337.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-337.png,Lung_Opacity/masks/Lung_Opacity-337.png,True,G06036,test,f74172b3a7cab1b0c0dceda9d6a78b83,c42e13486c7af2fa
Lung_Opacity-3370.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3370.png,Lung_Opacity/masks/Lung_Opacity-3370.png,True,G06037,test,8b7860c6354ddc824590a8bce990ef7c,c13d1a920b1bbdbc
Lung_Opacity-3371.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3371.png,Lung_Opacity/masks/Lung_Opacity-3371.png,True,G06038,test,0c8295c14527dd2849242beeabc902dc,880d161a5d5e7b67
Lung_Opacity-3372.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3372.png,Lung_Opacity/masks/Lung_Opacity-3372.png,True,G06039,train,b75c2fd4fb761eb7792cffe3730a006f,c03d73d20b0b5f4e
Lung_Opacity-3373.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3373.png,Lung_Opacity/masks/Lung_Opacity-3373.png,True,G06040,train,a19724b95aea7e3637c374a1b1974951,9d694e1c1a4b396d
Lung_Opacity-3374.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3374.png,Lung_Opacity/masks/Lung_Opacity-3374.png,True,G06041,train,1de3988fcc7dfff5efebb4779aa1f0c5,893a47181c6bbee9
Lung_Opacity-3375.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3375.png,Lung_Opacity/masks/Lung_Opacity-3375.png,True,G06042,train,41452f9f8876f474978590ca9d308697,dc2e035b4b597961
Lung_Opacity-3376.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3376.png,Lung_Opacity/masks/Lung_Opacity-3376.png,True,G06043,train,35fd9fc92e43c67d6afd467a0098acbd,d42e534b0a4a0fef
Lung_Opacity-3377.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3377.png,Lung_Opacity/masks/Lung_Opacity-3377.png,True,G06044,train,06ee2bc1ae5aefb492156ed28032fde4,887dd61e585a7c49
Lung_Opacity-3378.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3378.png,Lung_Opacity/masks/Lung_Opacity-3378.png,True,G06045,train,026a3b38248d60e72463dad8630fdc16,8b5b141a5a1f1f1e
Lung_Opacity-3379.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3379.png,Lung_Opacity/masks/Lung_Opacity-3379.png,True,G06046,train,b8216e0667b2101c6df4b2331e55bb25,d12b96187b591b1e
Lung_Opacity-338.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-338.png,Lung_Opacity/masks/Lung_Opacity-338.png,True,G06047,train,a7f6a2e0833bdba79d48f41e05d37617,c42f050f0f0c6f6f
Lung_Opacity-3380.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3380.png,Lung_Opacity/masks/Lung_Opacity-3380.png,True,G06048,val,9fabb3269d30a38bebb2d34a69741926,ca3d714b4e6c2c1e
Lung_Opacity-3381.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3381.png,Lung_Opacity/masks/Lung_Opacity-3381.png,True,G06049,train,27877a6f261c49013b6701d7d0f66f92,c23dd11fd6cd0c32
Lung_Opacity-3382.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3382.png,Lung_Opacity/masks/Lung_Opacity-3382.png,True,G06050,train,6a63d0d67c2233f000396f725b5b8ae4,d82f869e1a130b3f
Lung_Opacity-3383.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3383.png,Lung_Opacity/masks/Lung_Opacity-3383.png,True,G06051,train,2e05a1f0c6671c9242f4452b7fde3bdd,c56c861b1d1e4e4f
Lung_Opacity-3384.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3384.png,Lung_Opacity/masks/Lung_Opacity-3384.png,True,G06052,train,7d6001d8b5629c6bc342c8014f21d223,d83f161c501f5d96
Lung_Opacity-3385.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3385.png,Lung_Opacity/masks/Lung_Opacity-3385.png,True,G06053,train,a5c7f199be5340cd70c13c01852a9960,95685e5f58585963
Lung_Opacity-3386.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3386.png,Lung_Opacity/masks/Lung_Opacity-3386.png,True,G06054,val,882f60deaf06305e59b3a87e0f538301,9138161e5e1b5b5b
Lung_Opacity-3387.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3387.png,Lung_Opacity/masks/Lung_Opacity-3387.png,True,G06055,val,e9ad013eff9b148e4f00efea0bc37a81,c02dc31f1c79793a
Lung_Opacity-3388.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3388.png,Lung_Opacity/masks/Lung_Opacity-3388.png,True,G06056,test,7695fb6feafe8624c2a17e3eaf8c08e2,c50f0f4d4d4b4b5a
Lung_Opacity-3389.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3389.png,Lung_Opacity/masks/Lung_Opacity-3389.png,True,G06057,test,8561b5c653b6f0fb75ef5cc096c62e4b,c03d3bf0c6cf7032
Lung_Opacity-339.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-339.png,Lung_Opacity/masks/Lung_Opacity-339.png,True,G06058,test,a6c894fc132d112dccd225fa30cd2d5f,9568074f1f6d0f68
Lung_Opacity-3390.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3390.png,Lung_Opacity/masks/Lung_Opacity-3390.png,True,G06059,train,dc1cae7c9ba0108d2b74ec7e7d648c85,cc3d424b0b4e2f4f
Lung_Opacity-3391.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3391.png,Lung_Opacity/masks/Lung_Opacity-3391.png,True,G06060,train,cbbccb5ef18727b15d255228f394e41a,c87f801f1c464f5d
Lung_Opacity-3392.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3392.png,Lung_Opacity/masks/Lung_Opacity-3392.png,True,G06061,train,0c699a1ee48309e65e58bca57a3e9259,d42c43421e0eef7d
Lung_Opacity-3393.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3393.png,Lung_Opacity/masks/Lung_Opacity-3393.png,True,G06062,train,1cc9b9c23e1e096237310b0bf6b7b5bf,cc1d974a4c5a69e9
Lung_Opacity-3394.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3394.png,Lung_Opacity/masks/Lung_Opacity-3394.png,True,G06063,train,88369120ef0a42594570863a890d4ba0,b96a061c0e797979
Lung_Opacity-3395.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3395.png,Lung_Opacity/masks/Lung_Opacity-3395.png,True,G06064,train,8a721bffb2a46301b805706c58d0b6a9,817b4e06de1efa48
Lung_Opacity-3396.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3396.png,Lung_Opacity/masks/Lung_Opacity-3396.png,True,G06065,val,6f0f65dbc0bf550bf47b33a7c44615e6,cd3e924b1d680e9d
Lung_Opacity-3397.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3397.png,Lung_Opacity/masks/Lung_Opacity-3397.png,True,G06066,train,c43ad37e46231f3cbf3ac18e31d6c2d9,c42e1a9d4b4e6d3c
Lung_Opacity-3398.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3398.png,Lung_Opacity/masks/Lung_Opacity-3398.png,True,G06067,train,7237351a44398af12896907061c09e97,cd2ed35b4c4e0f24
Lung_Opacity-3399.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3399.png,Lung_Opacity/masks/Lung_Opacity-3399.png,True,G06068,train,dc3605e433ac1581c9e96d6bd484875e,c1e93e963a902eda
Lung_Opacity-34.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-34.png,Lung_Opacity/masks/Lung_Opacity-34.png,True,G06069,val,09c05b3089808fcbc5e5beedb6e0081e,c93d12124b5c5f5b
Lung_Opacity-340.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-340.png,Lung_Opacity/masks/Lung_Opacity-340.png,True,G06070,val,68734dfc7146cfc3035893b9e07dd807,d93d04c71a1ee94b
Lung_Opacity-3400.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3400.png,Lung_Opacity/masks/Lung_Opacity-3400.png,True,G06071,train,10059f2a12d19fec0d0c1c837982a5f3,856806167b6d567d
Lung_Opacity-3401.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3401.png,Lung_Opacity/masks/Lung_Opacity-3401.png,True,G06072,train,76a3ae08bb4e198f0a9ff7fe4091be05,81782747aa16575f
Lung_Opacity-3402.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3402.png,Lung_Opacity/masks/Lung_Opacity-3402.png,True,G06073,train,42b027abe9f54269d2f38ca8b1533cb8,917bd794dcc4c119
Lung_Opacity-3403.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3403.png,Lung_Opacity/masks/Lung_Opacity-3403.png,True,G06074,train,a2baec7ebb2609c399338494b2acdf4a,c13dc3561f144b1f
Lung_Opacity-3404.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3404.png,Lung_Opacity/masks/Lung_Opacity-3404.png,True,G06075,train,6f4a5b33e7c121c57cacafe43a5c5ce4,8178159e5e1a7a5b
Lung_Opacity-3405.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3405.png,Lung_Opacity/masks/Lung_Opacity-3405.png,True,G06076,test,13fca877b23919fd48e3b58fd462a9f1,d528d2171f6c1b1e
Lung_Opacity-3406.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3406.png,Lung_Opacity/masks/Lung_Opacity-3406.png,True,G06077,val,524258834a508492a7be6e78382adb73,dd2ad31b4d4c0e4e
Lung_Opacity-3407.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3407.png,Lung_Opacity/masks/Lung_Opacity-3407.png,True,G06078,test,e24f8ed35179ec96ceea78be717bb065,9138561e1e1b1f3d
Lung_Opacity-3408.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3408.png,Lung_Opacity/masks/Lung_Opacity-3408.png,True,G06079,test,87782539bb10b42875ff5895ac7b43b2,816b5e947ad96b90
Lung_Opacity-3409.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3409.png,Lung_Opacity/masks/Lung_Opacity-3409.png,True,G06080,train,a5affd4a11f7750d4d55280ba3d47af0,c42f16571a597d68
Lung_Opacity-341.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-341.png,Lung_Opacity/masks/Lung_Opacity-341.png,True,G06081,train,6bb18568c39033eb08615931de57ba54,c8197b18199f9f92
Lung_Opacity-3410.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3410.png,Lung_Opacity/masks/Lung_Opacity-3410.png,True,G06082,train,173e20fe8ec44ee03d26c4e1be1f91f6,dc314b0f6d343c35
Lung_Opacity-3411.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3411.png,Lung_Opacity/masks/Lung_Opacity-3411.png,True,G06083,test,4feaa28da35e9498086234483c7375e6,d12b56065a1d1f5b
Lung_Opacity-3412.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3412.png,Lung_Opacity/masks/Lung_Opacity-3412.png,True,G06084,val,c807c6eb0d4604faaffc9ffa8d431e74,d42a4b4f0e0e2f3d
Lung_Opacity-3413.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3413.png,Lung_Opacity/masks/Lung_Opacity-3413.png,True,G06085,val,e21cfdceebe4f95162570a83c5a8b6a4,d42e054f0c5e2e7e
Lung_Opacity-3414.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3414.png,Lung_Opacity/masks/Lung_Opacity-3414.png,True,G06086,test,0bc313d000a63215585c1d70c468855b,c03d4fc3d21b4d87
Lung_Opacity-3415.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3415.png,Lung_Opacity/masks/Lung_Opacity-3415.png,True,G06087,test,b8d72cf7083cbd3dd79d86287931b319,c447430b666f7ce4
Lung_Opacity-3416.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3416.png,Lung_Opacity/masks/Lung_Opacity-3416.png,True,G06088,train,12f5e2d6b6c6b43dc84554e0b766e7a2,9034141b3337bfab
Lung_Opacity-3417.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3417.png,Lung_Opacity/masks/Lung_Opacity-3417.png,True,G06089,train,6f568d20a2015da5c1c86a6d40b092b0,d92d865f185b4b38
Lung_Opacity-3418.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3418.png,Lung_Opacity/masks/Lung_Opacity-3418.png,True,G06090,train,ca25a07e7ed6039481b42d77c0270da8,c43f550e0a2f2e6e
Lung_Opacity-3419.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3419.png,Lung_Opacity/masks/Lung_Opacity-3419.png,True,G06091,train,ee8672ef929db64d665644b965661e9c,db3d590a2b393d30
Lung_Opacity-342.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-342.png,Lung_Opacity/masks/Lung_Opacity-342.png,True,G06092,train,5119c8c3338f4910443e10834f24f6cd,dd2ad24bc712496d
Lung_Opacity-3420.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3420.png,Lung_Opacity/masks/Lung_Opacity-3420.png,True,G06093,train,b88e1eed36365ca45046dbc8b8a8673c,d52d42574b4f0a5d
Lung_Opacity-3421.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3421.png,Lung_Opacity/masks/Lung_Opacity-3421.png,True,G06094,train,490f60e6425f867ceb1cc7059d59a5d7,c939c61b155d194f
Lung_Opacity-3422.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3422.png,Lung_Opacity/masks/Lung_Opacity-3422.png,True,G06095,train,ec33199938763a626b362da3f4abe20f,8d2e071f1c4d1a5f
Lung_Opacity-3423.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3423.png,Lung_Opacity/masks/Lung_Opacity-3423.png,True,G06096,train,90961bdebfc9357bcecacfe174a980ad,915e141b3b3aaeac
Lung_Opacity-3424.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3424.png,Lung_Opacity/masks/Lung_Opacity-3424.png,True,G06097,train,2f764bf7679725cb7773da3179317c71,d13d061b5be87839
Lung_Opacity-3425.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3425.png,Lung_Opacity/masks/Lung_Opacity-3425.png,True,G06098,train,7d640a530746feea9eea08c7956ba8e0,de3dd10b410f2e4e
Lung_Opacity-3426.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3426.png,Lung_Opacity/masks/Lung_Opacity-3426.png,True,G06099,train,1893041b09375b52335a57a20c251fec,99784e1459497de5
Lung_Opacity-3427.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3427.png,Lung_Opacity/masks/Lung_Opacity-3427.png,True,G06100,train,76670c2a74737c22c88a2eb82524feb9,d42a434f4e4e4f4e
Lung_Opacity-3428.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3428.png,Lung_Opacity/masks/Lung_Opacity-3428.png,True,G06101,train,ab8bfb68fae1448766e30cbca28360a1,8d6d4516534f1e5c
Lung_Opacity-3429.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3429.png,Lung_Opacity/masks/Lung_Opacity-3429.png,True,G06102,train,7d1e594ffc7c753d6d198498b7922414,95680697de5a1d1b
Lung_Opacity-343.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-343.png,Lung_Opacity/masks/Lung_Opacity-343.png,True,G06103,train,7a7a4f72326eb59627626a71dfd5f02d,c03d1a175b1f4e1d
Lung_Opacity-3430.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3430.png,Lung_Opacity/masks/Lung_Opacity-3430.png,True,G06104,train,62aa0a9b1ad5dea155d1d3a99605b765,dd2a524b474f4c1d
Lung_Opacity-3431.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3431.png,Lung_Opacity/masks/Lung_Opacity-3431.png,True,G06105,train,8c92ba28048eb5017e7b796b953eb0fd,937a041f1a7b7b28
Lung_Opacity-3432.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3432.png,Lung_Opacity/masks/Lung_Opacity-3432.png,True,G06106,train,2ec27187951fec53c386c59c5039b0e0,cc3c574b18ece4a9
Lung_Opacity-3433.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3433.png,Lung_Opacity/masks/Lung_Opacity-3433.png,True,G06107,train,69464e21f9f32fa7d40a2d78dc59191a,89789716184e5f5b
Lung_Opacity-3434.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3434.png,Lung_Opacity/masks/Lung_Opacity-3434.png,True,G06108,train,dad58836a6abdcf97d1ef7d1afbae36e,dc2cc30b2c687aee
Lung_Opacity-3435.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3435.png,Lung_Opacity/masks/Lung_Opacity-3435.png,True,G06109,train,5a5c86ff2e8d6dc52ddf414936efd432,d93fc61ba8a2b8a2
Lung_Opacity-3436.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3436.png,Lung_Opacity/masks/Lung_Opacity-3436.png,True,G06110,train,10c105f217152d9a3c99eb844fea60b5,81795e7849ed15a6
Lung_Opacity-3437.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3437.png,Lung_Opacity/masks/Lung_Opacity-3437.png,True,G06111,train,ef5a2f36ddc90c2b61567c07e8530cba,8c6f429b0f4da43e
Lung_Opacity-3438.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3438.png,Lung_Opacity/masks/Lung_Opacity-3438.png,True,G06112,train,8cc35dd094589d45db781fc4d5482b80,dd2e50434f4e4b0f
Lung_Opacity-3439.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3439.png,Lung_Opacity/masks/Lung_Opacity-3439.png,True,G06113,val,92238bcd319d21340c0e14f14013bf3e,95600e1d197d3f5a
Lung_Opacity-344.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-344.png,Lung_Opacity/masks/Lung_Opacity-344.png,True,G06114,test,40bfcedab51dc39d9795f0831867badd,c52a4f4a4b4d6f0b
Lung_Opacity-3440.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3440.png,Lung_Opacity/masks/Lung_Opacity-3440.png,True,G06115,val,d22a3ca1e31013b190685f644b83f0fd,817a47e5c19b9a6a
Lung_Opacity-3441.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3441.png,Lung_Opacity/masks/Lung_Opacity-3441.png,True,G06116,val,0ed83f737ba0b25a74ec382ff4417eaf,98585e5e3aa5a5a5
Lung_Opacity-3442.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3442.png,Lung_Opacity/masks/Lung_Opacity-3442.png,True,G06117,test,de5f226c5ef2d7680c288dd43ddfdffa,99396f16920f4b39
Lung_Opacity-3443.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3443.png,Lung_Opacity/masks/Lung_Opacity-3443.png,True,G06118,test,ab5190d5bf6e679a9a0a575580621e06,917e471c5a59494f
Lung_Opacity-3444.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3444.png,Lung_Opacity/masks/Lung_Opacity-3444.png,True,G06119,train,e9d2c5adbcb2b1c5a99e4199f53a923c,d56a07550f4e0e4f
Lung_Opacity-3445.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3445.png,Lung_Opacity/masks/Lung_Opacity-3445.png,True,G06120,train,28d9c44705413660b3c6b0ae1d8d7a03,d039c3574e4a5e59
Lung_Opacity-3446.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3446.png,Lung_Opacity/masks/Lung_Opacity-3446.png,True,G06121,val,f460bd07d8d66136267550840d7c10ce,d46ca79353c38393
Lung_Opacity-3447.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3447.png,Lung_Opacity/masks/Lung_Opacity-3447.png,True,G06122,train,de2d0bcaa087578691fecc310bd36774,cc2c170f4e5c6c6e
Lung_Opacity-3448.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3448.png,Lung_Opacity/masks/Lung_Opacity-3448.png,True,G06123,train,e7cdac7ad876c80540156b1ac371cadd,c03de2c30f7c6c3c
Lung_Opacity-3449.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3449.png,Lung_Opacity/masks/Lung_Opacity-3449.png,True,G06124,train,1f05c3bd1b5809df3385c18696555771,907d87071a1d1f1f
Lung_Opacity-345.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-345.png,Lung_Opacity/masks/Lung_Opacity-345.png,True,G06125,test,18e2694001ff4774f349c0c2fd87a1b6,d905461e591e1f6b
Lung_Opacity-3450.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3450.png,Lung_Opacity/masks/Lung_Opacity-3450.png,True,G06126,train,716276530ba798c69f4642ca1e3dd1c4,953ea2c3c3179b0b
Lung_Opacity-3451.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3451.png,Lung_Opacity/masks/Lung_Opacity-3451.png,True,G06127,train,51d00872bc6f93f97967149a5f4ad365,903ad6163d77151e
Lung_Opacity-3452.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3452.png,Lung_Opacity/masks/Lung_Opacity-3452.png,True,G06128,train,7b4a5c047532b25969cfda9c94c5df17,eb25cc062b6d076d
Lung_Opacity-3453.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3453.png,Lung_Opacity/masks/Lung_Opacity-3453.png,True,G06129,train,ddc343d805f3d182bbbfff151ef25dfe,ca2d5a4b4a6b4a6b
Lung_Opacity-3454.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3454.png,Lung_Opacity/masks/Lung_Opacity-3454.png,True,G06130,val,7a02dcf866c595be4d140ccedcf69527,856e071b4f4e7869
Lung_Opacity-3455.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3455.png,Lung_Opacity/masks/Lung_Opacity-3455.png,True,G06131,train,00596bf04b019868b66387b5c0a947bb,8c6d075b697ee032
Lung_Opacity-3456.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3456.png,Lung_Opacity/masks/Lung_Opacity-3456.png,True,G06132,val,53bf28d706477bd8da3816bda45fc8a8,996b461f491c4d1f
Lung_Opacity-3457.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3457.png,Lung_Opacity/masks/Lung_Opacity-3457.png,True,G06133,test,37d1a4c84e24954fbc499ab5fe75c97b,d42d420d481c3fff
Lung_Opacity-3458.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3458.png,Lung_Opacity/masks/Lung_Opacity-3458.png,True,G06134,val,5ad92cc9ae76e7e673222a167732467d,d03d0b093cdbd3d2
Lung_Opacity-3459.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3459.png,Lung_Opacity/masks/Lung_Opacity-3459.png,True,G06135,train,9c33686e18b86e9a50e7e9f0a16c8832,cc2d4f4a4c6cfce0
Lung_Opacity-346.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-346.png,Lung_Opacity/masks/Lung_Opacity-346.png,True,G06136,train,cede7be748b181afada46b6be41300b7,d42f065d1b496d3c
Lung_Opacity-3460.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3460.png,Lung_Opacity/masks/Lung_Opacity-3460.png,True,G06137,test,9acf1264957f7992772f484161445f56,cc300f4f3cf9e494
Lung_Opacity-3461.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3461.png,Lung_Opacity/masks/Lung_Opacity-3461.png,True,G06138,train,d44579cdbacf56f40af0410847bf6330,dc3d424b4f5d0a2d
Lung_Opacity-3462.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3462.png,Lung_Opacity/masks/Lung_Opacity-3462.png,True,G06139,train,b7bd2262e7d72b6ad8b9dc6c60f3429f,817a401e5a5b5b5f
Lung_Opacity-3463.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3463.png,Lung_Opacity/masks/Lung_Opacity-3463.png,True,G06140,train,a7419f027ba5bc13b1175a8af73150c1,c03d4b0c2b3d3e1f
Lung_Opacity-3464.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3464.png,Lung_Opacity/masks/Lung_Opacity-3464.png,True,G06141,train,8d441dcd3ba8d0f45f85edd5dde4668d,dc3d430b0e4f3c3c
Lung_Opacity-3465.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3465.png,Lung_Opacity/masks/Lung_Opacity-3465.png,True,G06142,train,caae512ba64e6afcb2d10f5ea7113a97,dd28525e0f29393d
Lung_Opacity-3466.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3466.png,Lung_Opacity/masks/Lung_Opacity-3466.png,True,G06143,train,f333edee95c7506ca5c7e00f7d295c59,947f475a1e490d4e
Lung_Opacity-3467.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3467.png,Lung_Opacity/masks/Lung_Opacity-3467.png,True,G06144,train,f97691c0342243701dd1c66c18c7bf2e,c02d5d195d1ab1bb
Lung_Opacity-3468.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3468.png,Lung_Opacity/masks/Lung_Opacity-3468.png,True,G06145,train,66396f27b878c6597190de0cdb3c2c81,d90ac70b1b0b2fab
Lung_Opacity-3469.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3469.png,Lung_Opacity/masks/Lung_Opacity-3469.png,True,G06146,train,981ea726d77afab90d9f85c4ab2de5e7,dd3f21ca3889299d
Lung_Opacity-347.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-347.png,Lung_Opacity/masks/Lung_Opacity-347.png,True,G06147,train,2eb345599b566c63e740c6549a794931,d93f460e1c6139bc
Lung_Opacity-3470.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3470.png,Lung_Opacity/masks/Lung_Opacity-3470.png,True,G06148,train,c75c0d1da16c181f0cbdbbc69a661da5,91605e5a793939ad
Lung_Opacity-3471.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3471.png,Lung_Opacity/masks/Lung_Opacity-3471.png,True,G06149,train,ee063671a9248a632135475d57a0708b,d42f475f084d4f0e
Lung_Opacity-3472.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3472.png,Lung_Opacity/masks/Lung_Opacity-3472.png,True,G06150,train,a4c66b1018b53acf5167e6ab20f409e7,997f4618433ce9c5
Lung_Opacity-3473.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3473.png,Lung_Opacity/masks/Lung_Opacity-3473.png,True,G06151,train,1680ada53d0e7c169d1d923d3434c62d,91356d9c39993d98
Lung_Opacity-3474.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3474.png,Lung_Opacity/masks/Lung_Opacity-3474.png,True,G06152,val,59090d558e47b7496cfc3906f5fa3345,9875475a571a591e
Lung_Opacity-3475.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3475.png,Lung_Opacity/masks/Lung_Opacity-3475.png,True,G06153,train,2172989a2e42b4a4291d05ad430c8f81,c42f3e399c4d0d4d
Lung_Opacity-3476.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3476.png,Lung_Opacity/masks/Lung_Opacity-3476.png,True,G06154,train,3ab52d9c788405fb696b66190e8511a6,97e08c4c3254de77
Lung_Opacity-3477.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3477.png,Lung_Opacity/masks/Lung_Opacity-3477.png,True,G06155,train,86d1b9e4404bc096c61a27e9702b0b62,c7264a0f2f0e0f2f
Lung_Opacity-3478.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3478.png,Lung_Opacity/masks/Lung_Opacity-3478.png,True,G06156,val,8a2c10006a9e88732c8f846ab3be4b50,d47dc30b4d0e4b2d
Lung_Opacity-3479.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3479.png,Lung_Opacity/masks/Lung_Opacity-3479.png,True,G06157,test,d2688692dd5e0595dd93999111d22140,dc2cd2434f4f494e
Lung_Opacity-348.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-348.png,Lung_Opacity/masks/Lung_Opacity-348.png,True,G06158,val,2988c34b587b92a339eb42acd8679f17,dd2c42165e683df2
Lung_Opacity-3480.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3480.png,Lung_Opacity/masks/Lung_Opacity-3480.png,True,G06159,test,f5c650d6a34666b6edfeba0d4212a861,d039535b591a3a3b
Lung_Opacity-3481.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3481.png,Lung_Opacity/masks/Lung_Opacity-3481.png,True,G06160,test,12c9c54a667268268013c1717ca1e32f,dc3d5056c2c31f5a
Lung_Opacity-3482.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3482.png,Lung_Opacity/masks/Lung_Opacity-3482.png,True,G06161,train,efc8129ed77c551694a854171f9221e3,dc2ad34d4e0f0a4f
Lung_Opacity-3483.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3483.png,Lung_Opacity/masks/Lung_Opacity-3483.png,True,G06162,train,367194d8e21b7e28caca593c0c13de90,d17f4619594b1d49
Lung_Opacity-3484.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3484.png,Lung_Opacity/masks/Lung_Opacity-3484.png,True,G06163,train,8913d85ea1b3d2046a786ee9b233960b,d42a575b0b2b0f0f
Lung_Opacity-3485.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3485.png,Lung_Opacity/masks/Lung_Opacity-3485.png,True,G06164,train,60c3b28547ecad497726fcb8c1da3dad,8d3fa0d11b9b091f
Lung_Opacity-3486.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3486.png,Lung_Opacity/masks/Lung_Opacity-3486.png,True,G06165,val,4f4f215a4b7f68d3473752667aeee499,837a0c111b5f7b5a
Lung_Opacity-3487.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3487.png,Lung_Opacity/masks/Lung_Opacity-3487.png,True,G06166,test,5cd0c442ec57a7dba52d8bf520eea29e,897907145a5b797a
Lung_Opacity-3488.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3488.png,Lung_Opacity/masks/Lung_Opacity-3488.png,True,G06167,train,c37bee0414ff6b4258bd009d6c19f535,ddaa565f421f440e
Lung_Opacity-3489.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3489.png,Lung_Opacity/masks/Lung_Opacity-3489.png,True,G06168,test,fed31c52c4c88611c41cb45488a92810,91695e5a59714be1
Lung_Opacity-349.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-349.png,Lung_Opacity/masks/Lung_Opacity-349.png,True,G06169,train,be4f6c6d5c6e856b7538fb539ac804b9,d42d12175b0b5e1f
Lung_Opacity-3490.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3490.png,Lung_Opacity/masks/Lung_Opacity-3490.png,True,G06170,train,e02ba023a9eda1eba96d43a448543e1c,cd36e0898fe61636
Lung_Opacity-3491.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3491.png,Lung_Opacity/masks/Lung_Opacity-3491.png,True,G06171,train,41f8438b9bc428dbc58afd46407c5f54,d835c3c34b4f4b4a
Lung_Opacity-3492.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3492.png,Lung_Opacity/masks/Lung_Opacity-3492.png,True,G06172,test,c75203ab31ba0848bee113c5691a56b0,99781f1f1a5918f2
Lung_Opacity-3493.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3493.png,Lung_Opacity/masks/Lung_Opacity-3493.png,True,G06173,train,5c7abe6fb8a5e2130d1e31b9430c629a,9b78045e1a7b7b48
Lung_Opacity-3494.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3494.png,Lung_Opacity/masks/Lung_Opacity-3494.png,True,G06174,train,89ff764dd571ac4f21c5672fa1f2cf1d,d42fc70f4c0b7869
Lung_Opacity-3495.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3495.png,Lung_Opacity/masks/Lung_Opacity-3495.png,True,G06175,train,dc588d931b9bf0ccfcd77da495848bd6,996bc7184b1dc99c
Lung_Opacity-3496.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3496.png,Lung_Opacity/masks/Lung_Opacity-3496.png,True,G06176,train,6556d86ee7516230bd462a27f66f3b3f,ac25870f455fc14f
Lung_Opacity-3497.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3497.png,Lung_Opacity/masks/Lung_Opacity-3497.png,True,G06177,train,2e214e63a76d808a8cb7af5c107f2d95,9939c786964a4f1d
Lung_Opacity-3498.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3498.png,Lung_Opacity/masks/Lung_Opacity-3498.png,True,G06178,train,e1d739a6660a499650afafd2fa8466c1,81e806103f5f795f
Lung_Opacity-3499.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3499.png,Lung_Opacity/masks/Lung_Opacity-3499.png,True,G06179,train,a5969a74845bd882f42bff65cb6b0bb0,d03d420e4f4b4f4f
Lung_Opacity-35.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-35.png,Lung_Opacity/masks/Lung_Opacity-35.png,True,G06180,train,3c6b15608d6882afa5a50be10dd6ba23,817816031b7ded6b
Lung_Opacity-350.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-350.png,Lung_Opacity/masks/Lung_Opacity-350.png,True,G06181,train,6690b3179a4608a2f436f00adee17e4a,d035430a4b4d5b7f
Lung_Opacity-3500.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3500.png,Lung_Opacity/masks/Lung_Opacity-3500.png,True,G06182,val,37a48051191738b8d5b3f7816051297f,d56ec10f4b4e6c29
Lung_Opacity-3501.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3501.png,Lung_Opacity/masks/Lung_Opacity-3501.png,True,G06183,val,4c4248e662cdc67c93eb9f98a9d46f30,817886173b0bee1f
Lung_Opacity-3502.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3502.png,Lung_Opacity/masks/Lung_Opacity-3502.png,True,G06184,train,6074b5453d2024b54888ff8df4b2ad26,dc0d5a0b0e2d3d3d
Lung_Opacity-3503.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3503.png,Lung_Opacity/masks/Lung_Opacity-3503.png,True,G06185,train,a7297702a9cd391045c43d62f23ca768,802f025f4d5c7e7a
Lung_Opacity-3504.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3504.png,Lung_Opacity/masks/Lung_Opacity-3504.png,True,G06186,train,568c6f12ebaa5a4db4df9c69f4eabdaa,cc3e83411b4e1f5d
Lung_Opacity-3505.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3505.png,Lung_Opacity/masks/Lung_Opacity-3505.png,True,G06187,train,09e6a7973d18a253bc6cf4e934533c79,d52e0e104d6f6c6e
Lung_Opacity-3506.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3506.png,Lung_Opacity/masks/Lung_Opacity-3506.png,True,G06188,train,7624bf86c505783bdcf416ea2a00fb84,997d061a786d6978
Lung_Opacity-3507.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3507.png,Lung_Opacity/masks/Lung_Opacity-3507.png,True,G06189,train,0bd96d6a2a67b3bb2b23d8916feef724,d16a471e0a5f1c2f
Lung_Opacity-3508.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3508.png,Lung_Opacity/masks/Lung_Opacity-3508.png,True,G06190,train,fb476d48f570f508002dec87e2c9e649,8368151f5c1d1f7a
Lung_Opacity-3509.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3509.png,Lung_Opacity/masks/Lung_Opacity-3509.png,True,G06191,train,46699cf5be65a7362866245a8dbde2e7,c52e4b085f4f4d4d
Lung_Opacity-351.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-351.png,Lung_Opacity/masks/Lung_Opacity-351.png,True,G06192,train,804ae5ff5f0a695b446bc1dc18a321fa,d06f431a1d6b16f2
Lung_Opacity-3510.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3510.png,Lung_Opacity/masks/Lung_Opacity-3510.png,True,G06193,val,62aa57aa0b787a45952540ce3ac6bb7c,91696e9439cf68d2
Lung_Opacity-3511.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3511.png,Lung_Opacity/masks/Lung_Opacity-3511.png,True,G06194,val,53f7f0ba49e74eec4961f49f2e3fb2b9,d079481a1fbd35b4
Lung_Opacity-3512.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3512.png,Lung_Opacity/masks/Lung_Opacity-3512.png,True,G06195,train,517013a47bfc930f595767d7d4835ae1,c47fcb2d542ca01f
Lung_Opacity-3513.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3513.png,Lung_Opacity/masks/Lung_Opacity-3513.png,True,G06196,train,788cf3d87fce5d7ad5de5b40c7421a38,d93942d61e4829bf
Lung_Opacity-3514.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3514.png,Lung_Opacity/masks/Lung_Opacity-3514.png,True,G06197,test,5ba1400d9b18cfd0e9d4ffc255a4466b,d52f0539387a7878
Lung_Opacity-3515.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3515.png,Lung_Opacity/masks/Lung_Opacity-3515.png,True,G06198,test,77c4e94d24332c2c34aeb9626cdd03ea,806bb797564a4d3c
Lung_Opacity-3516.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3516.png,Lung_Opacity/masks/Lung_Opacity-3516.png,True,G06199,val,4a0a097b8d19dc98dc08ee2eba43994c,d53f831f070f0707
Lung_Opacity-3517.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3517.png,Lung_Opacity/masks/Lung_Opacity-3517.png,True,G06200,test,ae982b391ba5b54a3a64b66c3fdcc31d,9179465e593b3938
Lung_Opacity-3518.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3518.png,Lung_Opacity/masks/Lung_Opacity-3518.png,True,G06201,train,5296e74ba3c8508559a804fd6b958822,95281d1a5f5b5959
Lung_Opacity-3519.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3519.png,Lung_Opacity/masks/Lung_Opacity-3519.png,True,G06202,train,416f7f5b693d6e5f2b97433f2f81d45e,9178071a5f2a7d59
Lung_Opacity-352.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-352.png,Lung_Opacity/masks/Lung_Opacity-352.png,True,G06203,train,b8b16efa8d6b0136917e8f726f764463,897d85961e4ac71e
Lung_Opacity-3520.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3520.png,Lung_Opacity/masks/Lung_Opacity-3520.png,True,G06204,train,ed57cf7875ee14802d8f410522e4ac67,d03d4a1a2b2b2d3f
Lung_Opacity-3521.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3521.png,Lung_Opacity/masks/Lung_Opacity-3521.png,True,G06205,train,04b638cb875d179fbe0b78a9916dc88b,9179445e9ac969bc
Lung_Opacity-3522.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3522.png,Lung_Opacity/masks/Lung_Opacity-3522.png,True,G06206,train,efc1dbd505bcd78496a53d1257ce99d1,c27d170a0e2f0f4f
Lung_Opacity-3523.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3523.png,Lung_Opacity/masks/Lung_Opacity-3523.png,True,G06207,train,213889cb6ef98c8aaa2fac12a9949eae,c95c1513175b1e6d
Lung_Opacity-3524.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3524.png,Lung_Opacity/masks/Lung_Opacity-3524.png,True,G06208,train,39edf1f094764042826f7dc4efef1fb2,812096165b4f5b7f
Lung_Opacity-3525.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3525.png,Lung_Opacity/masks/Lung_Opacity-3525.png,True,G06209,train,a81b9f138e1467a78ceb44979407ecf9,c12e161a0d5e5f4f
Lung_Opacity-3526.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3526.png,Lung_Opacity/masks/Lung_Opacity-3526.png,True,G06210,train,25c65af238d2e07db13fd5148c594ad0,807bc784fe14c717
Lung_Opacity-3527.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3527.png,Lung_Opacity/masks/Lung_Opacity-3527.png,True,G06211,train,c0ed282b81bc41acc9207da999a95d75,c02b97060f1f1f1f
Lung_Opacity-3528.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3528.png,Lung_Opacity/masks/Lung_Opacity-3528.png,True,G06212,train,7df54574a87a8695ad32d491ff4a0939,c97b965a1d1c5919
Lung_Opacity-3529.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3529.png,Lung_Opacity/masks/Lung_Opacity-3529.png,True,G06213,train,d9ef7e0fe7a3a65a02056b63d70739bc,817e2b070a5b2b5f
Lung_Opacity-353.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-353.png,Lung_Opacity/masks/Lung_Opacity-353.png,True,G06214,train,5d9da8ca113a4a367c8adc2a07a07807,d07d07431f0f0f0f
Lung_Opacity-3530.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3530.png,Lung_Opacity/masks/Lung_Opacity-3530.png,True,G06215,train,17195cc15c3ff71a713773f6b14829c6,802e07479f0f1f5d
Lung_Opacity-3531.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3531.png,Lung_Opacity/masks/Lung_Opacity-3531.png,True,G06216,val,9c8d928a5cc9be89477dbbc75c19429c,912bc6961a5b5e5c
Lung_Opacity-3532.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3532.png,Lung_Opacity/masks/Lung_Opacity-3532.png,True,G06217,val,03edfe63ba3b580f382567ec0af890fe,cd3f16411e3c4979
Lung_Opacity-3533.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3533.png,Lung_Opacity/masks/Lung_Opacity-3533.png,True,G06218,val,800902f08be5a46baa874b1fc7fbcc62,c939061e1ccb3bb9
Lung_Opacity-3534.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3534.png,Lung_Opacity/masks/Lung_Opacity-3534.png,True,G06219,test,cb9c2e5cc3dd8642ba85b2c7a391f85d,c0add7129149d57d
Lung_Opacity-3535.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3535.png,Lung_Opacity/masks/Lung_Opacity-3535.png,True,G06220,test,5deee13f8d935249ba74d470a64ec8c1,d1379410133fbcb9
Lung_Opacity-3536.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3536.png,Lung_Opacity/masks/Lung_Opacity-3536.png,True,G06221,test,5b3af8604fbcc20922a672c3b8b58578,9939e7c69e54424b
Lung_Opacity-3537.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3537.png,Lung_Opacity/masks/Lung_Opacity-3537.png,True,G06222,train,60b09983e4ef9499f0bcbefe1643c015,c0744a5b3a6d3d1d
Lung_Opacity-3538.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3538.png,Lung_Opacity/masks/Lung_Opacity-3538.png,True,G06223,train,eee2889782ee790d662cec42217dd644,dc2a930b0d7c6e69
Lung_Opacity-3539.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3539.png,Lung_Opacity/masks/Lung_Opacity-3539.png,True,G06224,train,496e796adc24fb881d2c0987a697298a,d43d4b0f2d053c3e
Lung_Opacity-354.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-354.png,Lung_Opacity/masks/Lung_Opacity-354.png,True,G06225,train,ace9816e25f274456a2da5accb90d71e,843d07831f0f4f4f
Lung_Opacity-3540.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3540.png,Lung_Opacity/masks/Lung_Opacity-3540.png,True,G06226,train,d639101595296bcb60def78e1a701a6b,816d345217937add
Lung_Opacity-3541.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3541.png,Lung_Opacity/masks/Lung_Opacity-3541.png,True,G06227,train,c22756b9e0db09deb5e15e65a76442f9,807fc64a5f5b0b1a
Lung_Opacity-3542.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3542.png,Lung_Opacity/masks/Lung_Opacity-3542.png,True,G06228,train,12769115aa3912aff739b2ad932c6fe5,c06d471b392abd2e
Lung_Opacity-3543.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3543.png,Lung_Opacity/masks/Lung_Opacity-3543.png,True,G06229,train,072221a9b1afe6c7304b78abe6b58597,946e875d0d4c2e4f
Lung_Opacity-3544.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3544.png,Lung_Opacity/masks/Lung_Opacity-3544.png,True,G06230,train,3b95d546c8682a3ee1592f34c046fb4c,81391f585a5a59fa
Lung_Opacity-3545.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3545.png,Lung_Opacity/masks/Lung_Opacity-3545.png,True,G06231,train,e672569ad1e1580bb2aa91b947f7d583,c6274ac94566b576
Lung_Opacity-3546.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3546.png,Lung_Opacity/masks/Lung_Opacity-3546.png,True,G06232,test,4130dbf4be3f8f8f7d6d5189b2e723a8,d52d3c5293536363
Lung_Opacity-3547.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3547.png,Lung_Opacity/masks/Lung_Opacity-3547.png,True,G06233,train,f5e9d054f82febfeb30fd2dde6ede792,dd2853130f4f1b1b
Lung_Opacity-3548.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3548.png,Lung_Opacity/masks/Lung_Opacity-3548.png,True,G06234,train,18e4ac4e23739a36fb35a9961cbd921f,c139125159f9e9b9
Lung_Opacity-3549.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3549.png,Lung_Opacity/masks/Lung_Opacity-3549.png,True,G06235,train,19e808803bfd29cab91b0ac51e63b1f7,8378009f9f1fd01f
Lung_Opacity-355.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-355.png,Lung_Opacity/masks/Lung_Opacity-355.png,True,G06236,train,57fd74bb7357ed71567a707054d499a6,881f165f5878f8e8
Lung_Opacity-3550.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3550.png,Lung_Opacity/masks/Lung_Opacity-3550.png,True,G06237,val,2c32992c0327d755da17ef5db0b876e4,9979051e193bfaa8
Lung_Opacity-3551.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3551.png,Lung_Opacity/masks/Lung_Opacity-3551.png,True,G06238,val,b48c362f41132a0b3e56e38c6a8f5d41,d56a514b48eae5e1
Lung_Opacity-3552.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3552.png,Lung_Opacity/masks/Lung_Opacity-3552.png,True,G06239,test,1d3e894d9bb1f7b3d0e353f1d39d1612,c12f365b094d4e5e
Lung_Opacity-3553.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3553.png,Lung_Opacity/masks/Lung_Opacity-3553.png,True,G06240,val,71827b242f4c84624ec49f2a37ff15df,d62eab01515b5d1b
Lung_Opacity-3554.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3554.png,Lung_Opacity/masks/Lung_Opacity-3554.png,True,G06241,test,dfbdabe2d334af4f798dfbdfad571e15,d03d6fc30f0e2e0d
Lung_Opacity-3555.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3555.png,Lung_Opacity/masks/Lung_Opacity-3555.png,True,G06242,train,239fae9da6ac639a42e6d0e4eaf86dc4,dc2f42574f4a0e4e
Lung_Opacity-3556.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3556.png,Lung_Opacity/masks/Lung_Opacity-3556.png,True,G06243,train,3bb5a2697c1ec00456349d329ffffe50,dd2ad34f4d4d0c1c
Lung_Opacity-3557.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3557.png,Lung_Opacity/masks/Lung_Opacity-3557.png,True,G06244,train,7d4104ac47ec81933c39b27ec77edcfb,813bd21a195b5b5b
Lung_Opacity-3558.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3558.png,Lung_Opacity/masks/Lung_Opacity-3558.png,True,G06245,train,b757be960eab0c369f9704bdb20d8bdd,c13b560b19293fad
Lung_Opacity-3559.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3559.png,Lung_Opacity/masks/Lung_Opacity-3559.png,True,G06246,train,078790a111f9b961283e2870b9ea38ac,9d781e5f28e7a2a0
Lung_Opacity-356.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-356.png,Lung_Opacity/masks/Lung_Opacity-356.png,True,G06247,train,6b5fc3b7bd11cc5a8d8ae6149f0b9b02,8d2a0f4f686be1f0
Lung_Opacity-3560.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3560.png,Lung_Opacity/masks/Lung_Opacity-3560.png,True,G06248,val,c622d0123b2eefed3cbeec32777ecf17,886fc63a096d694f
Lung_Opacity-3561.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3561.png,Lung_Opacity/masks/Lung_Opacity-3561.png,True,G06249,train,4110a9ba9709d79ae9805c872c99b146,9f611f041e1e1e1f
Lung_Opacity-3562.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3562.png,Lung_Opacity/masks/Lung_Opacity-3562.png,True,G06250,train,dc160ef4400aecaaafc7495584ef994f,d92f461e096d06be
Lung_Opacity-3563.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3563.png,Lung_Opacity/masks/Lung_Opacity-3563.png,True,G06251,train,5d80cb48c08241efbb175c1f213b044f,c46f014b0f5e6e6c
Lung_Opacity-3564.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3564.png,Lung_Opacity/masks/Lung_Opacity-3564.png,True,G06252,train,6e5951a2490db2533170a09d613b1ec1,917c07861f1f0f0f
Lung_Opacity-3565.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3565.png,Lung_Opacity/masks/Lung_Opacity-3565.png,True,G06253,train,13d1a756f89266ee781b6d7059ef0536,9979851e1ec45d5a
Lung_Opacity-3566.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3566.png,Lung_Opacity/masks/Lung_Opacity-3566.png,True,G06254,test,d0a4d57b30f5eaacae271b99cf694c8d,817f4f5719092d4d
Lung_Opacity-3567.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3567.png,Lung_Opacity/masks/Lung_Opacity-3567.png,True,G06255,train,e2c26a2fe381b92f4d9a8c1c9f06a556,897c561b597938e4
Lung_Opacity-3568.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3568.png,Lung_Opacity/masks/Lung_Opacity-3568.png,True,G06256,train,f4e23bbf47281d6ec9cdf0a2bed74e1f,c97b14124b193b7b
Lung_Opacity-3569.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3569.png,Lung_Opacity/masks/Lung_Opacity-3569.png,True,G06257,train,f0be0bbe6915aff008109260eaf63232,81381e485f6b7d69
Lung_Opacity-357.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-357.png,Lung_Opacity/masks/Lung_Opacity-357.png,True,G06258,val,383a9a0705b8eb3a76ba58fb74aec9a0,c03d4f4a1d993d9a
Lung_Opacity-3570.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3570.png,Lung_Opacity/masks/Lung_Opacity-3570.png,True,G06259,val,ef948fc580b04fdff2c24b77ba0cf514,c56c1e184d6ca6fa
Lung_Opacity-3571.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3571.png,Lung_Opacity/masks/Lung_Opacity-3571.png,True,G06260,test,1a63c5d627eb438c8e2e0f79b8a9135e,cc3f124b1c6c2dae
Lung_Opacity-3572.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3572.png,Lung_Opacity/masks/Lung_Opacity-3572.png,True,G06261,test,5b24110ab38875fe9ccd69e29dcd65ac,c02dc30dcf0e6f0f
Lung_Opacity-3573.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3573.png,Lung_Opacity/masks/Lung_Opacity-3573.png,True,G06262,train,72b6c1d801e78bf6adcae749fd10065d,cd3e530f0c0c8f8f
Lung_Opacity-3574.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3574.png,Lung_Opacity/masks/Lung_Opacity-3574.png,True,G06263,train,15524f7afaa08a435e4f61b7ac68a90e,c42f435b4d4d0e4f
Lung_Opacity-3575.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3575.png,Lung_Opacity/masks/Lung_Opacity-3575.png,True,G06264,train,b2940fa44788861ec9238072546d4ad8,907c47431e7ce53c
Lung_Opacity-3576.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3576.png,Lung_Opacity/masks/Lung_Opacity-3576.png,True,G06265,train,ca4f7a8d1bc4c11c307071fe8226d64c,985b021a1b39b7be
Lung_Opacity-3577.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3577.png,Lung_Opacity/masks/Lung_Opacity-3577.png,True,G06266,train,94f65626030e34cdb5f3f9fc0a3423f3,817caf1e0b1d1a1f
Lung_Opacity-3578.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3578.png,Lung_Opacity/masks/Lung_Opacity-3578.png,True,G06267,train,38778e292afb88eb63d7cb3910a924ba,85780e1b1b7d7961
Lung_Opacity-3579.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3579.png,Lung_Opacity/masks/Lung_Opacity-3579.png,True,G06268,train,54684fefe93d031e7793d91eebd8e109,913a121db7bec4a5
Lung_Opacity-358.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-358.png,Lung_Opacity/masks/Lung_Opacity-358.png,True,G06269,train,219bd2bae22c1b72c8dac548cf2ac127,837a041d1d5f5c5e
Lung_Opacity-3580.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3580.png,Lung_Opacity/masks/Lung_Opacity-3580.png,True,G06270,train,b0f3719aaea418e13c239281e6aeaaab,9979071e1a486d7d
Lung_Opacity-3581.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3581.png,Lung_Opacity/masks/Lung_Opacity-3581.png,True,G06271,train,1c263eaa6316620384fd15ef7ba3da03,c53e0713479f1b15
Lung_Opacity-3582.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3582.png,Lung_Opacity/masks/Lung_Opacity-3582.png,True,G06272,train,5ba839cd1d875e1dfba277fe01ffad51,81791e1a5a79696d
Lung_Opacity-3583.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3583.png,Lung_Opacity/masks/Lung_Opacity-3583.png,True,G06273,train,b533f00b3787c119c0601a5b529224f1,816f071a5e1e7969
Lung_Opacity-3584.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3584.png,Lung_Opacity/masks/Lung_Opacity-3584.png,True,G06274,train,e247d9c584afa314cb4e3e731940579c,c52f861b190d4f4f
Lung_Opacity-3585.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3585.png,Lung_Opacity/masks/Lung_Opacity-3585.png,True,G06275,val,f7847fdbdbc20f8c2ab3e47e7275052d,dc2a531f0a1b2d3e
Lung_Opacity-3586.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3586.png,Lung_Opacity/masks/Lung_Opacity-3586.png,True,G06276,train,405c325fcd81820c3a69abbb92c7a932,917a4e961a3bab38
Lung_Opacity-3587.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3587.png,Lung_Opacity/masks/Lung_Opacity-3587.png,True,G06277,val,767def447cd0745b53d3c635dd24a5e4,806c86d2ea696d5f
Lung_Opacity-3588.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3588.png,Lung_Opacity/masks/Lung_Opacity-3588.png,True,G06278,test,d7cf70e014a9bf61b3b74275f9aef4a3,996087561a57bf0b
Lung_Opacity-3589.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3589.png,Lung_Opacity/masks/Lung_Opacity-3589.png,True,G06279,val,b7fdf7cb8632476451a3df13144981ae,903f401b5b7b3933
Lung_Opacity-359.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-359.png,Lung_Opacity/masks/Lung_Opacity-359.png,True,G06280,test,b82af22454c6112d4fa60aca7328b58e,807f5416105b7f3b
Lung_Opacity-3590.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3590.png,Lung_Opacity/masks/Lung_Opacity-3590.png,True,G06281,test,1d90b6cc091406efeacc8e858b08ac9d,d96b041e4e4f584f
Lung_Opacity-3591.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3591.png,Lung_Opacity/masks/Lung_Opacity-3591.png,True,G06282,train,398ff503c30083daa3e2cf269547df52,9d29c2171e4b1d5e
Lung_Opacity-3592.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3592.png,Lung_Opacity/masks/Lung_Opacity-3592.png,True,G06283,test,af44251cc710113d6529213f47ab28ee,d03bc78e1c687b38
Lung_Opacity-3593.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3593.png,Lung_Opacity/masks/Lung_Opacity-3593.png,True,G06284,train,1f236bfafcb4307b4628a2ee06857321,c42d571a1a0b5f7c
Lung_Opacity-3594.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3594.png,Lung_Opacity/masks/Lung_Opacity-3594.png,True,G06285,train,698e0d664e15842ac33bed1c1b30481a,913b8b191d616bf2
Lung_Opacity-3595.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3595.png,Lung_Opacity/masks/Lung_Opacity-3595.png,True,G06286,train,0592ea07941bcf0214a0d5710997a667,c07d420b0f3d3c3f
Lung_Opacity-3596.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3596.png,Lung_Opacity/masks/Lung_Opacity-3596.png,True,G06287,train,1d6999468a33282a7ac67df48381acfd,884fce0c4d71f1ec
Lung_Opacity-3597.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3597.png,Lung_Opacity/masks/Lung_Opacity-3597.png,True,G06288,train,df995ae5b7d0a20000445797e2da5e11,d43c534b0f0f0f0f
Lung_Opacity-3598.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3598.png,Lung_Opacity/masks/Lung_Opacity-3598.png,True,G06289,train,99c8f466f9c7ac82c87e433f6d0e78ed,df68d41e5a596858
Lung_Opacity-3599.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3599.png,Lung_Opacity/masks/Lung_Opacity-3599.png,True,G06290,train,92223bfc8f454a8662526806548d055e,c075fe42ad4af42a
Lung_Opacity-36.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-36.png,Lung_Opacity/masks/Lung_Opacity-36.png,True,G06291,train,80342d9e0a48e24f7aad0c9a4c9faed9,de1d490a3c15b6b6
Lung_Opacity-360.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-360.png,Lung_Opacity/masks/Lung_Opacity-360.png,True,G06292,train,1bfe03e0cb4437f58482ea4eea73c3d4,dc3d420b4c6d7d64
Lung_Opacity-3600.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3600.png,Lung_Opacity/masks/Lung_Opacity-3600.png,True,G06293,train,efc9bb7c3418eefb06f140f9bb1ee24f,d03d7d438f0f0d0e
Lung_Opacity-3601.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3601.png,Lung_Opacity/masks/Lung_Opacity-3601.png,True,G06294,train,e4c4d3b3ccdf5e6f741af8607a66a35f,9139424b5a5a6f6b
Lung_Opacity-3602.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3602.png,Lung_Opacity/masks/Lung_Opacity-3602.png,True,G06295,train,5c177854f64c1437210ac5416f80fe1a,c02d5f082d2f3c3f
Lung_Opacity-3603.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3603.png,Lung_Opacity/masks/Lung_Opacity-3603.png,True,G06296,train,b0e596a2fb32eb88c0b843194e3d7022,dc3d935b2c5a381a
Lung_Opacity-3604.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3604.png,Lung_Opacity/masks/Lung_Opacity-3604.png,True,G06297,test,45da636bcf1c6c36aaa118346fd96d7e,dc2d40930f485f5f
Lung_Opacity-3605.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3605.png,Lung_Opacity/masks/Lung_Opacity-3605.png,True,G06298,val,42fc85ef1bff2fd4a7a4a05b0aab8d38,817fd4163e193e32
Lung_Opacity-3606.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3606.png,Lung_Opacity/masks/Lung_Opacity-3606.png,True,G06299,train,3d336aee91c440f466443783625f5cef,903f131a181a5fdf
Lung_Opacity-3607.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3607.png,Lung_Opacity/masks/Lung_Opacity-3607.png,True,G06300,val,1419486fbffd978a811d32fdb0558690,df2b12db299129d4
Lung_Opacity-3608.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3608.png,Lung_Opacity/masks/Lung_Opacity-3608.png,True,G06301,val,7bf563f3047e42db7dbce1e7b8ccd4c3,dd0a934b0e6cf8f0
Lung_Opacity-3609.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3609.png,Lung_Opacity/masks/Lung_Opacity-3609.png,True,G06302,test,a42d6ef63ae019e399883b49d3b895bd,9d79c216c04f9ec9
Lung_Opacity-361.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-361.png,Lung_Opacity/masks/Lung_Opacity-361.png,True,G06303,train,c566c4de9cbb838de72903057f1d6751,d02d2b423fda39f0
Lung_Opacity-3610.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3610.png,Lung_Opacity/masks/Lung_Opacity-3610.png,True,G06304,train,940342c33a7c5a8148cfae24da98968c,d42cb7171e1c2d0f
Lung_Opacity-3611.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3611.png,Lung_Opacity/masks/Lung_Opacity-3611.png,True,G06305,train,e637f0799225c79936bf0181bcd28256,d02f531a5d5b1a4e
Lung_Opacity-3612.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3612.png,Lung_Opacity/masks/Lung_Opacity-3612.png,True,G06306,train,4277ce1ed090f752c26124dfd2fb2fc2,c03df3494a58497f
Lung_Opacity-3613.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3613.png,Lung_Opacity/masks/Lung_Opacity-3613.png,True,G06307,train,38dc9037a8dcde32802abc6fc3e0c26d,dc2d40c30f1ce9ed
Lung_Opacity-3614.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3614.png,Lung_Opacity/masks/Lung_Opacity-3614.png,True,G06308,train,4cafaeb1ac13a010ff50a3f15321c821,976a5ca3831c7167
Lung_Opacity-3615.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3615.png,Lung_Opacity/masks/Lung_Opacity-3615.png,True,G06309,train,d4ad5dec8b2504514f0c6b15fba73620,d42ec7550c4e2e4f
Lung_Opacity-3616.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3616.png,Lung_Opacity/masks/Lung_Opacity-3616.png,True,G06310,train,a4caf8502e914cb34f2a6ed7344664e2,9568071e2b5e786d
Lung_Opacity-3617.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3617.png,Lung_Opacity/masks/Lung_Opacity-3617.png,True,G06311,train,c8921efe801b99a9171ed399844b2349,c968161a4d5f195f
Lung_Opacity-3618.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3618.png,Lung_Opacity/masks/Lung_Opacity-3618.png,True,G06312,train,789a2753a44587d2b03e6ee25fd1a53a,c93916561b195b4f
Lung_Opacity-3619.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3619.png,Lung_Opacity/masks/Lung_Opacity-3619.png,True,G06313,train,11cbb99826fa9e5bbe8372c68f2c77a6,c03df54d5c1c7d0c
Lung_Opacity-362.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-362.png,Lung_Opacity/masks/Lung_Opacity-362.png,True,G06314,test,528d11c54b51e66ed11b8c240c90dc56,cd0e4e0f19eda0b6
Lung_Opacity-3620.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3620.png,Lung_Opacity/masks/Lung_Opacity-3620.png,True,G06315,train,0f01acb86b49f370436af3ba70a0d002,d42b43594d6f0d1d
Lung_Opacity-3621.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3621.png,Lung_Opacity/masks/Lung_Opacity-3621.png,True,G06316,train,e7efaf8a8e1a480fb0104cf3d6b9437a,887da516161e3a3f
Lung_Opacity-3622.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3622.png,Lung_Opacity/masks/Lung_Opacity-3622.png,True,G06317,train,181d2be5b8105195b66a760d152c2caa,99694e131b4f3a39
Lung_Opacity-3623.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3623.png,Lung_Opacity/masks/Lung_Opacity-3623.png,True,G06318,val,9f5f5050047980d9dd2bd543793af326,81680c1f1f4f3b5b
Lung_Opacity-3624.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3624.png,Lung_Opacity/masks/Lung_Opacity-3624.png,True,G06319,train,8aabd0e482c3e43cc3d7919ceae3fadf,c5aff4135290db15
Lung_Opacity-3625.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3625.png,Lung_Opacity/masks/Lung_Opacity-3625.png,True,G06320,train,5af942f7315c1d9da11c2011aafe1929,8d680f565e5a5ae8
Lung_Opacity-3626.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3626.png,Lung_Opacity/masks/Lung_Opacity-3626.png,True,G06321,val,89b9ef8ce8b2e13ae3c4136e36e2c778,c93b47c797174305
Lung_Opacity-3627.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3627.png,Lung_Opacity/masks/Lung_Opacity-3627.png,True,G06322,val,13ac68a82e65cd83e2d7792ac5e7b05d,987b471e495839e9
Lung_Opacity-3628.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3628.png,Lung_Opacity/masks/Lung_Opacity-3628.png,True,G06323,test,f838bdf83a8d57b8f45fdbb12385df77,c02d1b434f2f2f3c
Lung_Opacity-3629.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3629.png,Lung_Opacity/masks/Lung_Opacity-3629.png,True,G06324,train,f6aafb84ca673512861ad3c8f5b8b66e,983f965a185a497e
Lung_Opacity-363.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-363.png,Lung_Opacity/masks/Lung_Opacity-363.png,True,G06325,test,dcc44259d4fa2bf62dbf32881a91eea8,873a579f501ada58
Lung_Opacity-3630.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3630.png,Lung_Opacity/masks/Lung_Opacity-3630.png,True,G06326,train,0d388a505f8e77a404a85075aeac977a,9d93627494636d8d
Lung_Opacity-3631.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3631.png,Lung_Opacity/masks/Lung_Opacity-3631.png,True,G06327,train,a1f45fa1605e869a03f849dc521df6ed,81394ac6d65e5e5a
Lung_Opacity-3632.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3632.png,Lung_Opacity/masks/Lung_Opacity-3632.png,True,G06328,train,ac7e8ec1a881e01e55017324a76a3d0e,983ac3021f595f1f
Lung_Opacity-3633.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3633.png,Lung_Opacity/masks/Lung_Opacity-3633.png,True,G06329,train,f0e358b628f9169c1021fb4cc8d6d58a,817f0616565f5b4a
Lung_Opacity-3634.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3634.png,Lung_Opacity/masks/Lung_Opacity-3634.png,True,G06330,train,c3e4d75cb9377c3add31da9fb800f8a3,d52f17035a2c7e29
Lung_Opacity-3635.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3635.png,Lung_Opacity/masks/Lung_Opacity-3635.png,True,G06331,train,47af1d66aba8b8f4b19b023ececefa8b,9979c7041e475b5a
Lung_Opacity-3636.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3636.png,Lung_Opacity/masks/Lung_Opacity-3636.png,True,G06332,train,fb2067f824470948f04cd7e755de808e,c96f14911a1f3d55
Lung_Opacity-3637.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3637.png,Lung_Opacity/masks/Lung_Opacity-3637.png,True,G06333,train,8f6f1e8054478595889b8f410a1c979b,886a075b787d7570
Lung_Opacity-3638.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3638.png,Lung_Opacity/masks/Lung_Opacity-3638.png,True,G06334,train,a7e6878ee41340afe0d3f911f8b05754,992dc41b1d4d1a9f
Lung_Opacity-3639.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3639.png,Lung_Opacity/masks/Lung_Opacity-3639.png,True,G06335,train,f955283f41db86786f921c73033a3da8,dd0c4e58787ce8f0
Lung_Opacity-364.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-364.png,Lung_Opacity/masks/Lung_Opacity-364.png,True,G06336,val,55a583ea4d69cdb1cbfcd35e71eac2e3,c42c7b414b6b2f6a
Lung_Opacity-3640.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3640.png,Lung_Opacity/masks/Lung_Opacity-3640.png,True,G06337,test,e71b025d30e6b511133c8c43de7e3d0a,d42e031d0d475e5f
Lung_Opacity-3641.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3641.png,Lung_Opacity/masks/Lung_Opacity-3641.png,True,G06338,train,265dfc7560fe523e444e3dea5d7d7086,986f02377139673a
Lung_Opacity-3642.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3642.png,Lung_Opacity/masks/Lung_Opacity-3642.png,True,G06339,train,f64a1a979958e74ad795f26cd7e593d1,9839c74c49471f57
Lung_Opacity-3643.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3643.png,Lung_Opacity/masks/Lung_Opacity-3643.png,True,G06340,val,7cec715851a3bf5edc6c50205b1f6d9c,a83dc3831f4b2f0e
Lung_Opacity-3644.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3644.png,Lung_Opacity/masks/Lung_Opacity-3644.png,True,G06341,val,59785704972bb3d89d5621278bcf0f45,91e8061b194d5e7f
Lung_Opacity-3645.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3645.png,Lung_Opacity/masks/Lung_Opacity-3645.png,True,G06342,test,6fcc8bfce23c0611c199fcf8c69fcc7f,c4644f6a64e5e393
Lung_Opacity-3646.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3646.png,Lung_Opacity/masks/Lung_Opacity-3646.png,True,G06343,test,e77567be0f2a5762434f8ddafaaae3ce,81794e865a5f885f
Lung_Opacity-3647.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3647.png,Lung_Opacity/masks/Lung_Opacity-3647.png,True,G06344,train,89b08dc6fd289b062bb06e902cd7249b,985f961a59797ac0
Lung_Opacity-3648.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3648.png,Lung_Opacity/masks/Lung_Opacity-3648.png,True,G06345,train,b6a69d06200fc028243062011ff4b534,d42a55131e5e1e5e
Lung_Opacity-3649.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3649.png,Lung_Opacity/masks/Lung_Opacity-3649.png,True,G06346,train,bf5104d454a85d11dae436c4866a43ca,8068075e5f797c78
Lung_Opacity-365.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-365.png,Lung_Opacity/masks/Lung_Opacity-365.png,True,G06347,train,0d9f8aac4cce4df3dc657e943840f6d8,d03f921bdf031f44
Lung_Opacity-3650.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3650.png,Lung_Opacity/masks/Lung_Opacity-3650.png,True,G06348,train,d595e980a7671a921495a372d2d47f69,95680f56da58f878
Lung_Opacity-3651.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3651.png,Lung_Opacity/masks/Lung_Opacity-3651.png,True,G06349,train,aeee06b0dc7998e31caf160a3d6ea01c,9763535052727a5b
Lung_Opacity-3652.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3652.png,Lung_Opacity/masks/Lung_Opacity-3652.png,True,G06350,train,722b038939d9210dc27c2f8a11eb9fdb,937bc4951e169713
Lung_Opacity-3653.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3653.png,Lung_Opacity/masks/Lung_Opacity-3653.png,True,G06351,train,84de422fa7683640005c4bc2b86c5d01,d43dc3434a4b5b5a
Lung_Opacity-3654.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3654.png,Lung_Opacity/masks/Lung_Opacity-3654.png,True,G06352,train,1d1dbc32dec9d5d9b51bb5ab7dd3ea97,d42f830f0d4f4d4d
Lung_Opacity-3655.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3655.png,Lung_Opacity/masks/Lung_Opacity-3655.png,True,G06353,train,dd47e160d48cfa33a12eb479d2e7dfbf,cd38964b5f4879a1
Lung_Opacity-3656.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3656.png,Lung_Opacity/masks/Lung_Opacity-3656.png,True,G06354,train,af9189454ad4df3f5d7776bbeb9f6961,817a065a5b58fada
Lung_Opacity-3657.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3657.png,Lung_Opacity/masks/Lung_Opacity-3657.png,True,G06355,train,1dbfc290527b1cd77e92906a0976acab,d03f49494e4e5c5e
Lung_Opacity-3658.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3658.png,Lung_Opacity/masks/Lung_Opacity-3658.png,True,G06356,val,d8250aa78c89fa60e32fd76f8565884b,d42a35574a0e6e6e
Lung_Opacity-3659.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3659.png,Lung_Opacity/masks/Lung_Opacity-3659.png,True,G06357,train,8e57f0fed0c60d5858394757cb6fcad7,c405530b2f2f6e6d
Lung_Opacity-366.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-366.png,Lung_Opacity/masks/Lung_Opacity-366.png,True,G06358,val,2692c12cb624e5f6ca4683816daecd95,d97e46186ce839e1
Lung_Opacity-3660.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3660.png,Lung_Opacity/masks/Lung_Opacity-3660.png,True,G06359,val,d25ded3d118c32f32e5aeba92ace7521,d42c56496c68f5e5
Lung_Opacity-3661.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3661.png,Lung_Opacity/masks/Lung_Opacity-3661.png,True,G06360,test,b4cd62a867b17f256ea14a59b967b8ef,d47f454b0f4e2c0e
Lung_Opacity-3662.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3662.png,Lung_Opacity/masks/Lung_Opacity-3662.png,True,G06361,test,8cfe2fedd29c09ac2a71ae9d47c67293,817a5051535b5b5f
Lung_Opacity-3663.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3663.png,Lung_Opacity/masks/Lung_Opacity-3663.png,True,G06362,test,fd574fe1c4a5507a296caf248daac8b3,817b04161f1f1f1f
Lung_Opacity-3664.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3664.png,Lung_Opacity/masks/Lung_Opacity-3664.png,True,G06363,train,d6d691e6cca037f534867eea5413124d,856e075f7938383a
Lung_Opacity-3665.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3665.png,Lung_Opacity/masks/Lung_Opacity-3665.png,True,G06364,train,c09ca42bea733694fa68f14cc001d924,99791c6267273636
Lung_Opacity-3666.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3666.png,Lung_Opacity/masks/Lung_Opacity-3666.png,True,G06365,train,834f280040126a38dc3288048a79fc61,d13b921a5f1e7838
Lung_Opacity-3667.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3667.png,Lung_Opacity/masks/Lung_Opacity-3667.png,True,G06366,train,4b91bbd6d5b5aa579857153b1e88eb76,c46d5a0a0f6d4d2f
Lung_Opacity-3668.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3668.png,Lung_Opacity/masks/Lung_Opacity-3668.png,True,G06367,train,af16a0d4efe0a853e3382628613c5aff,817b06165e1f5c5b
Lung_Opacity-3669.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3669.png,Lung_Opacity/masks/Lung_Opacity-3669.png,True,G06368,train,074f49c90fab82f5054e3bce80113b1e,d42b470d0c0f2fbb
Lung_Opacity-367.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-367.png,Lung_Opacity/masks/Lung_Opacity-367.png,True,G06369,train,7b663839902632dca2acaf1c776ce634,cc3f524f4f0c0e0f
Lung_Opacity-3670.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3670.png,Lung_Opacity/masks/Lung_Opacity-3670.png,True,G06370,val,1c9b4127d5114aab44c75c823d881d4d,d17910079bbb3939
Lung_Opacity-3671.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3671.png,Lung_Opacity/masks/Lung_Opacity-3671.png,True,G06371,train,db1e9e47da7e13addb370afb5b65a1fb,987b169e5a993a86
Lung_Opacity-3672.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3672.png,Lung_Opacity/masks/Lung_Opacity-3672.png,True,G06372,test,97d1c1aa0bda7e794766c4f95070680e,c43d83011b6ebbe9
Lung_Opacity-3673.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3673.png,Lung_Opacity/masks/Lung_Opacity-3673.png,True,G06373,train,e99ff63f773061b130a6d93cfa59b40f,d42b471f4f0c384f
Lung_Opacity-3674.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3674.png,Lung_Opacity/masks/Lung_Opacity-3674.png,True,G06374,train,17c979ef29b15fea3541851ae79938d4,c87d1443535e1e1f
Lung_Opacity-3675.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3675.png,Lung_Opacity/masks/Lung_Opacity-3675.png,True,G06375,val,b10bd697554e63b466ac545da3f8549f,d43f0341d5436e6e
Lung_Opacity-3676.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3676.png,Lung_Opacity/masks/Lung_Opacity-3676.png,True,G06376,test,0276fff0ad0c4b76aaf9696b0cef2473,c03f5a0d2d2d3537
Lung_Opacity-3677.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3677.png,Lung_Opacity/masks/Lung_Opacity-3677.png,True,G06377,train,780e87f98d632d52a2eb24f0d679c4da,dd3ad20b2a313c9e
Lung_Opacity-3678.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3678.png,Lung_Opacity/masks/Lung_Opacity-3678.png,True,G06378,train,31502a8349305888e86aba4030c0b0ea,d42d43d38b1f4d2c
Lung_Opacity-3679.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3679.png,Lung_Opacity/masks/Lung_Opacity-3679.png,True,G06379,train,4f99e1dd492763523cec645cd0df5fc3,c46f504b4b4b4f4b
Lung_Opacity-368.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-368.png,Lung_Opacity/masks/Lung_Opacity-368.png,True,G06380,test,e952fc154e26f612298c6f7b87cf541e,807f07861e1f971b
Lung_Opacity-3680.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3680.png,Lung_Opacity/masks/Lung_Opacity-3680.png,True,G06381,train,dc3fe24878524b03b7e4999e090c3757,d1281f0a3cb5b6f2
Lung_Opacity-3681.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3681.png,Lung_Opacity/masks/Lung_Opacity-3681.png,True,G06382,train,6fcd85a933de7182552fadcedb1583e6,dd29429f18c9add8
Lung_Opacity-3682.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3682.png,Lung_Opacity/masks/Lung_Opacity-3682.png,True,G06383,val,5e9a154800a27cd86763abc14522b416,d42d414b4e6e2e6e
Lung_Opacity-3683.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3683.png,Lung_Opacity/masks/Lung_Opacity-3683.png,True,G06384,train,e1f04e1751f096be3c87e50399119f6e,d43d53554d55c155
Lung_Opacity-3684.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3684.png,Lung_Opacity/masks/Lung_Opacity-3684.png,True,G06385,train,9796deadcb98af0852a8434af9933f1d,842e42575e496b3f
Lung_Opacity-3685.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3685.png,Lung_Opacity/masks/Lung_Opacity-3685.png,True,G06386,train,09cab2dec054eb8b60a30119f4520ed9,956ad65a4d2d3ca1
Lung_Opacity-3686.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3686.png,Lung_Opacity/masks/Lung_Opacity-3686.png,True,G06387,train,038ffdfe3e3b1e628127a3526b395320,986a47184fbcb8e5
Lung_Opacity-3687.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3687.png,Lung_Opacity/masks/Lung_Opacity-3687.png,True,G06388,train,dc09c5e2179e8dbb7b95187569c0e6da,8360859ddd1f505f
Lung_Opacity-3688.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3688.png,Lung_Opacity/masks/Lung_Opacity-3688.png,True,G06389,train,dc3807b62494cb8b45c82e4d244af0f8,853e1c1a595d6dcc
Lung_Opacity-3689.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3689.png,Lung_Opacity/masks/Lung_Opacity-3689.png,True,G06390,train,6b93f688b8ae363164d4ff3de9c4466d,97680e1b0d6d69e9
Lung_Opacity-369.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-369.png,Lung_Opacity/masks/Lung_Opacity-369.png,True,G06391,train,84df66791827c762b2e3350db9dd27b6,834a0f5f5a4b694b
Lung_Opacity-3690.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3690.png,Lung_Opacity/masks/Lung_Opacity-3690.png,True,G06392,train,4bd62330357d833a7f1678dbe852815d,817e061a5b4d787b
Lung_Opacity-3691.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3691.png,Lung_Opacity/masks/Lung_Opacity-3691.png,True,G06393,train,d2ffb019de4120f5d8758952847d6a55,d83f430d0c4adfc3
Lung_Opacity-3692.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3692.png,Lung_Opacity/masks/Lung_Opacity-3692.png,True,G06394,val,e5e657b6f30bd1976a2c693d370e9acf,956cc71a49c93ecc
Lung_Opacity-3693.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3693.png,Lung_Opacity/masks/Lung_Opacity-3693.png,True,G06395,train,317dc7b44be2608cb030a47f51ceac8d,c03f47131d4d0f5d
Lung_Opacity-3694.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3694.png,Lung_Opacity/masks/Lung_Opacity-3694.png,True,G06396,val,53869cc27b915de58a7f215cd776b1e6,817b1e1e185f591b
Lung_Opacity-3695.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3695.png,Lung_Opacity/masks/Lung_Opacity-3695.png,True,G06397,train,5e70b8e20811d51abf285c43baabdf84,987b425e1e4b1b1d
Lung_Opacity-3696.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3696.png,Lung_Opacity/masks/Lung_Opacity-3696.png,True,G06398,train,c20f4bfb2c4ccfa33b79513a25b7f0e7,807c075f0f5d3e0e
Lung_Opacity-3697.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3697.png,Lung_Opacity/masks/Lung_Opacity-3697.png,True,G06399,test,f34df9e046e1762f8d73260aacaf8b02,d43d4b434b426f4e
Lung_Opacity-3698.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3698.png,Lung_Opacity/masks/Lung_Opacity-3698.png,True,G06400,val,4c50facc2d16db209259d24adee5d4a8,807fc61a1c4d4d3f
Lung_Opacity-3699.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3699.png,Lung_Opacity/masks/Lung_Opacity-3699.png,True,G06401,test,f98472dfc4a830a317ef528229640ae6,c86db5d29393d292
Lung_Opacity-37.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-37.png,Lung_Opacity/masks/Lung_Opacity-37.png,True,G06402,train,0f5657c51058a66ca8f0283f9f75e383,847c2607971f9b1d
Lung_Opacity-370.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-370.png,Lung_Opacity/masks/Lung_Opacity-370.png,True,G06403,test,b97decfe6e204925f4cc5cca2bb683a7,917d041e5b0f5f0b
Lung_Opacity-3700.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3700.png,Lung_Opacity/masks/Lung_Opacity-3700.png,True,G06404,train,33aebe5e30a1b2c0908fa41aae256381,c03f7d4d142d0d1f
Lung_Opacity-3701.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3701.png,Lung_Opacity/masks/Lung_Opacity-3701.png,True,G06405,train,66986a62797465251ea856fdcc9b05af,887f5f1418593de8
Lung_Opacity-3702.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3702.png,Lung_Opacity/masks/Lung_Opacity-3702.png,True,G06406,val,e7273dd093661a61456c422c1e5b3162,916f461a4c5d5a4f
Lung_Opacity-3703.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3703.png,Lung_Opacity/masks/Lung_Opacity-3703.png,True,G06407,train,2a7dbce3dddd69c491dc78d4b0b12296,913be5c63c80e4f5
Lung_Opacity-3704.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3704.png,Lung_Opacity/masks/Lung_Opacity-3704.png,True,G06408,train,52227e35bac7afba6bd6aa3df931c74c,d12b0e9f5d287938
Lung_Opacity-3705.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3705.png,Lung_Opacity/masks/Lung_Opacity-3705.png,True,G06409,train,74064bdd3c3442d79cb0db87e325f418,d42d32d34bd02bd3
Lung_Opacity-3706.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3706.png,Lung_Opacity/masks/Lung_Opacity-3706.png,True,G06410,test,a0ce821b006ac12ef6611d48f0996196,d43d6bd21a4a4b1d
Lung_Opacity-3707.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3707.png,Lung_Opacity/masks/Lung_Opacity-3707.png,True,G06411,train,fef2f4c4d0505f89c33e9949c8405332,816a84575f4d565e
Lung_Opacity-3708.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3708.png,Lung_Opacity/masks/Lung_Opacity-3708.png,True,G06412,train,cba597aa85e0e7d29c93256e09584e9f,d87b435e1a1a2a1f
Lung_Opacity-3709.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3709.png,Lung_Opacity/masks/Lung_Opacity-3709.png,True,G06413,train,6a2bde0a01a360ddc95a1a676c29dbeb,d42cc7539a4daa5c
Lung_Opacity-371.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-371.png,Lung_Opacity/masks/Lung_Opacity-371.png,True,G06414,train,a9a0514dd8e185c2bfb4f4402dc77a1b,c0bc3fc32574c30f
Lung_Opacity-3710.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3710.png,Lung_Opacity/masks/Lung_Opacity-3710.png,True,G06415,train,a046f820884428d7e3c1a8545b602491,91686e16161a5f5f
Lung_Opacity-3711.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3711.png,Lung_Opacity/masks/Lung_Opacity-3711.png,True,G06416,train,f336242b16f55ca855503bcd4ece1050,d42f061f195d7a68
Lung_Opacity-3712.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3712.png,Lung_Opacity/masks/Lung_Opacity-3712.png,True,G06417,train,1278aa250e20df753d36535964db7b5d,d62e27560f082f2f
Lung_Opacity-3713.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3713.png,Lung_Opacity/masks/Lung_Opacity-3713.png,True,G06418,train,9522911b8ddec9e4e869a9ddb31d7310,d029431b3d383f97
Lung_Opacity-3714.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3714.png,Lung_Opacity/masks/Lung_Opacity-3714.png,True,G06419,val,78a2f831a67d58d871ef54224da665b3,dc2d570b2c2c2d2f
Lung_Opacity-3715.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3715.png,Lung_Opacity/masks/Lung_Opacity-3715.png,True,G06420,val,b4fc48351d777e22541eac72d46996dd,93725f1153d2d2ca
Lung_Opacity-3716.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3716.png,Lung_Opacity/masks/Lung_Opacity-3716.png,True,G06421,test,8a2d8da66c0dc66e2c11d3df99bb596a,d42d53454d087f6d
Lung_Opacity-3717.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3717.png,Lung_Opacity/masks/Lung_Opacity-3717.png,True,G06422,train,4b8105f296bcdcbdca410d3453822001,c56a070f454f4d4f
Lung_Opacity-3718.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3718.png,Lung_Opacity/masks/Lung_Opacity-3718.png,True,G06423,test,e1e44512bd4996602567fdec716d2cae,97680e576c686f61
Lung_Opacity-3719.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3719.png,Lung_Opacity/masks/Lung_Opacity-3719.png,True,G06424,train,33571462e29022decf479fcc7d17fa0c,817e060a5d5d5b4f
Lung_Opacity-372.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-372.png,Lung_Opacity/masks/Lung_Opacity-372.png,True,G06425,train,406beb1f25bdaea6410a1e7a2e1761cc,9d2cc21b4f4c1eea
Lung_Opacity-3720.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3720.png,Lung_Opacity/masks/Lung_Opacity-3720.png,True,G06426,train,6385ad20f394be172c09d8ab819c18df,c52d121e1a7d5a5b
Lung_Opacity-3721.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3721.png,Lung_Opacity/masks/Lung_Opacity-3721.png,True,G06427,train,0cd2d6a238a77fe304d1cb8c155fbf63,d82d4b1f0f0c3cbc
Lung_Opacity-3722.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3722.png,Lung_Opacity/masks/Lung_Opacity-3722.png,True,G06428,val,f268aa85243dc22499493edb7aeafbf2,c627420746e5f8f9
Lung_Opacity-3723.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3723.png,Lung_Opacity/masks/Lung_Opacity-3723.png,True,G06429,train,853e84f1b643b2000befd7b170807da0,c83d6b4b0e2d3d15
Lung_Opacity-3724.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3724.png,Lung_Opacity/masks/Lung_Opacity-3724.png,True,G06430,train,cf1a2051c1cf7ca86ef51e1c71ae1c43,916d871a1b171f5a
Lung_Opacity-3725.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3725.png,Lung_Opacity/masks/Lung_Opacity-3725.png,True,G06431,train,de2764d3176497e1bf8cb2d57ac3aadd,c03c1f430f3eb4e5
Lung_Opacity-3726.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3726.png,Lung_Opacity/masks/Lung_Opacity-3726.png,True,G06432,train,3ba70a9bc52af0e1b4cc7fca93e0ca6e,c969c6161bcd4f1a
Lung_Opacity-3727.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3727.png,Lung_Opacity/masks/Lung_Opacity-3727.png,True,G06433,train,e118c6abe840236d4f3aedcc1e5df475,b83ce3c79e0aa338
Lung_Opacity-3728.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3728.png,Lung_Opacity/masks/Lung_Opacity-3728.png,True,G06434,train,7560755a91897656d81dc41c66146626,9c2e434e4b4f0e4f
Lung_Opacity-3729.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3729.png,Lung_Opacity/masks/Lung_Opacity-3729.png,True,G06435,train,e4aafb49dc1ec6db288b347824c7c17c,d839431e49393dbc
Lung_Opacity-373.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-373.png,Lung_Opacity/masks/Lung_Opacity-373.png,True,G06436,train,673167ea089a8be3df3c70bd76b1aeae,d42dd3c30bcd025f
Lung_Opacity-3730.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3730.png,Lung_Opacity/masks/Lung_Opacity-3730.png,True,G06437,train,b3400d9803bec4b1fe8038f784de5949,d039425b3b1b3b1b
Lung_Opacity-3731.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3731.png,Lung_Opacity/masks/Lung_Opacity-3731.png,True,G06438,train,6deda2c1e28f7e28d1de50b4c59c23c0,9121cf1e9e1a5f19
Lung_Opacity-3732.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3732.png,Lung_Opacity/masks/Lung_Opacity-3732.png,True,G06439,test,c207f0c0fc8c3190ca79ac51b70c0bf5,d039521e6f3e6525
Lung_Opacity-3733.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3733.png,Lung_Opacity/masks/Lung_Opacity-3733.png,True,G06440,train,fd3b07790a69444c19102e863d5441cb,913a071a5d6a7a6b
Lung_Opacity-3734.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3734.png,Lung_Opacity/masks/Lung_Opacity-3734.png,True,G06441,val,451e235b1d297411e4fa4447b86e3466,8978155e5a797ae0
Lung_Opacity-3735.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3735.png,Lung_Opacity/masks/Lung_Opacity-3735.png,True,G06442,test,bacba5966bb63f8a6f117d9d52c18daa,957c2055cb13d75a
Lung_Opacity-3736.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3736.png,Lung_Opacity/masks/Lung_Opacity-3736.png,True,G06443,val,fa0a18812482980b12cd7f38f0f7b932,c17f0e0a1a5d7d38
Lung_Opacity-3737.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3737.png,Lung_Opacity/masks/Lung_Opacity-3737.png,True,G06444,train,71fa1b008d41c7386dcfc2722c71bd20,dc39d30e6d1d381c
Lung_Opacity-3738.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3738.png,Lung_Opacity/masks/Lung_Opacity-3738.png,True,G06445,train,f50ece60fc42b7b2f997077f6573b3d2,8978051e5f4d5eb8
Lung_Opacity-3739.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3739.png,Lung_Opacity/masks/Lung_Opacity-3739.png,True,G06446,test,4e61c8a10cfb402f023244da321adf6f,dc2c970f4d0f0f0d
Lung_Opacity-374.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-374.png,Lung_Opacity/masks/Lung_Opacity-374.png,True,G06447,val,ebcbfe3467df98ec39e9461ba08b9dee,c43c53471e1b6d39
Lung_Opacity-3740.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3740.png,Lung_Opacity/masks/Lung_Opacity-3740.png,True,G06448,train,796683c18b8bb88c98f5ff4923b64249,9578071e161e5e3b
Lung_Opacity-3741.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3741.png,Lung_Opacity/masks/Lung_Opacity-3741.png,True,G06449,train,43894aefffcdd9134e76bd0c4a20bc12,9f62475e54545674
Lung_Opacity-3742.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3742.png,Lung_Opacity/masks/Lung_Opacity-3742.png,True,G06450,train,eeb7b4f72bc6a3157597cac30e810b7f,8370955e5a5a7a5a
Lung_Opacity-3743.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3743.png,Lung_Opacity/masks/Lung_Opacity-3743.png,True,G06451,train,2babdd3eacaa296121d18286cfac3af2,9968061a5b5b5afa
Lung_Opacity-3744.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3744.png,Lung_Opacity/masks/Lung_Opacity-3744.png,True,G06452,train,57863f955ed72a81d811e7d58047ec50,81780d8e5e784fc7
Lung_Opacity-3745.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3745.png,Lung_Opacity/masks/Lung_Opacity-3745.png,True,G06453,train,cf9d2825f31c47b892315390dfa1ef04,c93a17c75a1c4c6b
Lung_Opacity-3746.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3746.png,Lung_Opacity/masks/Lung_Opacity-3746.png,True,G06454,train,6206cc1977df151bebc81e0f7a7a1f24,d42d831c6d3e38e3
Lung_Opacity-3747.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3747.png,Lung_Opacity/masks/Lung_Opacity-3747.png,True,G06455,train,7f9493c1507cc2b9c2cab20bfc02777b,897f8712185e1b5b
Lung_Opacity-3748.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3748.png,Lung_Opacity/masks/Lung_Opacity-3748.png,True,G06456,train,28eea700bc1b86ff329a1a69c66875cc,d029c3d61b4b0f7c
Lung_Opacity-3749.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3749.png,Lung_Opacity/masks/Lung_Opacity-3749.png,True,G06457,test,c39430439e764e4fb9f2364de3fd0c20,907dc2561b4b4b1f
Lung_Opacity-375.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-375.png,Lung_Opacity/masks/Lung_Opacity-375.png,True,G06458,train,bd1e1c6c7bfe0becead6b1c1d1ce265c,9938061f1c5b7d5a
Lung_Opacity-3750.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3750.png,Lung_Opacity/masks/Lung_Opacity-3750.png,True,G06459,test,fc8c6dcb8e8a949037ae43397df4e6a9,817e1c1c1e5a5b7a
Lung_Opacity-3751.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3751.png,Lung_Opacity/masks/Lung_Opacity-3751.png,True,G06460,train,a907a6c1ce3c6aa41408a3e2f2ba164c,d42f034f4cece8e2
Lung_Opacity-3752.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3752.png,Lung_Opacity/masks/Lung_Opacity-3752.png,True,G06461,train,a33906b18404975c80ab33111e5393ce,cd2a95134f165e0f
Lung_Opacity-3753.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3753.png,Lung_Opacity/masks/Lung_Opacity-3753.png,True,G06462,val,f237992eb5f7211b3e7bb93381f67404,817b4c185b3f3e25
Lung_Opacity-3754.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3754.png,Lung_Opacity/masks/Lung_Opacity-3754.png,True,G06463,val,a7ef3e51f825001e240a484dec3904b7,812c405f1f797d59
Lung_Opacity-3755.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3755.png,Lung_Opacity/masks/Lung_Opacity-3755.png,True,G06464,test,01bc91de0e0e9e5204ea3a7909009c67,c93f960e8a2aaeaa
Lung_Opacity-3756.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3756.png,Lung_Opacity/masks/Lung_Opacity-3756.png,True,G06465,train,35febf119718ae44d9cb153795775d06,d47ea50f6d1e2c0c
Lung_Opacity-3757.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3757.png,Lung_Opacity/masks/Lung_Opacity-3757.png,True,G06466,train,9a7e8b405a19eb91761e7c1718f00bc2,c43d935b0e0aeb2d
Lung_Opacity-3758.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3758.png,Lung_Opacity/masks/Lung_Opacity-3758.png,True,G06467,train,4e0e76949d1219067b3d7c13bad4e68c,8b7914121a5b5b5f
Lung_Opacity-3759.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3759.png,Lung_Opacity/masks/Lung_Opacity-3759.png,True,G06468,train,f077cd3e17b2fc9fc19e4d20d3c845f5,c039121a7f2d3d3d
Lung_Opacity-376.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-376.png,Lung_Opacity/masks/Lung_Opacity-376.png,True,G06469,train,b5e042dbaff71f0ea866b0335fe56b43,9c66431c453e7373
Lung_Opacity-3760.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3760.png,Lung_Opacity/masks/Lung_Opacity-3760.png,True,G06470,train,f7a4cae2847e29e505764c80fee22d42,c83d9643591d4b8f
Lung_Opacity-3761.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3761.png,Lung_Opacity/masks/Lung_Opacity-3761.png,True,G06471,train,ab11dd83199c2ad7c4ba5d3956440546,993b87831d1d585e
Lung_Opacity-3762.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3762.png,Lung_Opacity/masks/Lung_Opacity-3762.png,True,G06472,train,ce189b07e6ab0e9da5c75cd5a50a94bc,c82d47435a4d4d5f
Lung_Opacity-3763.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3763.png,Lung_Opacity/masks/Lung_Opacity-3763.png,True,G06473,train,7ccfb26b90b33a2e207c1cb8013e7b9d,85791e1e7b916985
Lung_Opacity-3764.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3764.png,Lung_Opacity/masks/Lung_Opacity-3764.png,True,G06474,train,d5ce200f2b873f2dbb1ddbbff550b188,8179521b39a937b5
Lung_Opacity-3765.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3765.png,Lung_Opacity/masks/Lung_Opacity-3765.png,True,G06475,train,20ff79e0b46fbb368b72a6db56309f53,907983571e0e4f2f
Lung_Opacity-3766.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3766.png,Lung_Opacity/masks/Lung_Opacity-3766.png,True,G06476,train,df30a277e921b12ffce89bb5bde51369,dc2d421b0b2d3dbc
Lung_Opacity-3767.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3767.png,Lung_Opacity/masks/Lung_Opacity-3767.png,True,G06477,train,e3940da1450a13dbd9055b1ef050ffe0,8978101b7b5bcec6
Lung_Opacity-3768.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3768.png,Lung_Opacity/masks/Lung_Opacity-3768.png,True,G06478,train,032d64380f2419d46ec8cc4e8a53dff8,95681e5b5a4c787a
Lung_Opacity-3769.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3769.png,Lung_Opacity/masks/Lung_Opacity-3769.png,True,G06479,val,30781ecc78e8cb6a971410e22697d20e,912e041e1e5f5a5f
Lung_Opacity-377.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-377.png,Lung_Opacity/masks/Lung_Opacity-377.png,True,G06480,val,00ca7fc1451001eab5e780870994f829,dc2c434b1f4e5c5c
Lung_Opacity-3770.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3770.png,Lung_Opacity/masks/Lung_Opacity-3770.png,True,G06481,val,fa037a4a6ebfbda4704338dbb03da680,c02fd61a5c4b4b4f
Lung_Opacity-3771.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3771.png,Lung_Opacity/masks/Lung_Opacity-3771.png,True,G06482,test,5082ddf9787253c442bb9101b8f30791,c42f434b2f6e6266
Lung_Opacity-3772.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3772.png,Lung_Opacity/masks/Lung_Opacity-3772.png,True,G06483,test,7e1588243eeef28d01c9265077f44084,856e06595f786e4c
Lung_Opacity-3773.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3773.png,Lung_Opacity/masks/Lung_Opacity-3773.png,True,G06484,test,360fc79969d39b9f9d67a8f44927060b,d42e070f1d7d686c
Lung_Opacity-3774.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3774.png,Lung_Opacity/masks/Lung_Opacity-3774.png,True,G06485,train,9662e37546ed26b829a9d83f9a94b910,c42e971b4f5e1a0b
Lung_Opacity-3775.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3775.png,Lung_Opacity/masks/Lung_Opacity-3775.png,True,G06486,train,625902340f6af42a0bd03ef4f1132c82,91694e1a4e1b795e
Lung_Opacity-3776.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3776.png,Lung_Opacity/masks/Lung_Opacity-3776.png,True,G06487,train,146af4e3e9d594ff9919aeaaef6b8159,95781e09194de5fa
Lung_Opacity-3777.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3777.png,Lung_Opacity/masks/Lung_Opacity-3777.png,True,G06488,train,28022b125d2158f8f0ba05258d23d571,8568161e1b2f3e3b
Lung_Opacity-3778.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3778.png,Lung_Opacity/masks/Lung_Opacity-3778.png,True,G06489,train,e4dfcf4ef9eeca555f01e37ae3a89380,907f471a495d1e47
Lung_Opacity-3779.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3779.png,Lung_Opacity/masks/Lung_Opacity-3779.png,True,G06490,train,fb0ea7a55ec7a8f4701a3a24d07c7faf,b939064e1a39595f
Lung_Opacity-378.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-378.png,Lung_Opacity/masks/Lung_Opacity-378.png,True,G06491,train,241beb8a44b5cf8a929718f9a9418ffe,de2dd1cb4c4a5d48
Lung_Opacity-3780.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3780.png,Lung_Opacity/masks/Lung_Opacity-3780.png,True,G06492,test,b89b39f3bd3d54f85360dfa62e65a80c,d46c434a2c3f353e
Lung_Opacity-3781.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3781.png,Lung_Opacity/masks/Lung_Opacity-3781.png,True,G06493,train,39d5e5ea699a815c373c2e7a4ca91db1,d47e6343130f2d2d
Lung_Opacity-3782.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3782.png,Lung_Opacity/masks/Lung_Opacity-3782.png,True,G06494,train,8b701336f28344771225877f28d67213,83688e9e17565e5a
Lung_Opacity-3783.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3783.png,Lung_Opacity/masks/Lung_Opacity-3783.png,True,G06495,train,3df309569d8158647d4d04d605c94e01,c138164a1f797ba5
Lung_Opacity-3784.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3784.png,Lung_Opacity/masks/Lung_Opacity-3784.png,True,G06496,train,dcf54891bc4a966a716c8161d43be7c0,985907165e1e5a5f
Lung_Opacity-3785.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3785.png,Lung_Opacity/masks/Lung_Opacity-3785.png,True,G06497,train,b1dc449c0660c4fe71df64fc1a53c91d,ee95f9a5f4a4a0a0
Lung_Opacity-3786.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3786.png,Lung_Opacity/masks/Lung_Opacity-3786.png,True,G06498,train,7e49031963f2f2e0af67e739733eeb07,91f8841eda1a39db
Lung_Opacity-3787.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3787.png,Lung_Opacity/masks/Lung_Opacity-3787.png,True,G06499,train,96116d351ceb06264949bcc500e470eb,d03d8b879f99888d
Lung_Opacity-3788.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3788.png,Lung_Opacity/masks/Lung_Opacity-3788.png,True,G06500,val,a4d49d34660bef9e4f240415cddb9c37,917a66e265c53a99
Lung_Opacity-3789.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3789.png,Lung_Opacity/masks/Lung_Opacity-3789.png,True,G06501,val,8c06caa89a34756e7dc12c8d7755c668,dd3cc21b4b441f6c
Lung_Opacity-379.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-379.png,Lung_Opacity/masks/Lung_Opacity-379.png,True,G06502,val,fc0ef8108743e132dfbcf43b2e2125b0,99e887d2dbb98889
Lung_Opacity-3790.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3790.png,Lung_Opacity/masks/Lung_Opacity-3790.png,True,G06503,train,c4e2f744107aeec5c4aa2bbf2f775a3a,c42e474f2c2ca5f3
Lung_Opacity-3791.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3791.png,Lung_Opacity/masks/Lung_Opacity-3791.png,True,G06504,test,f5e47364d9bdde350d3af838a1ceba0f,987947971a0ad71d
Lung_Opacity-3792.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3792.png,Lung_Opacity/masks/Lung_Opacity-3792.png,True,G06505,test,b9358a6b051e4a61c4db4074e19373f2,817f56465e4a5c5c
Lung_Opacity-3793.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3793.png,Lung_Opacity/masks/Lung_Opacity-3793.png,True,G06506,train,1e7eec4bd44eb731a87e168423f79f3a,8d4e07494ded78e1
Lung_Opacity-3794.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3794.png,Lung_Opacity/masks/Lung_Opacity-3794.png,True,G06507,train,0d1202dfb5399aa480e1e646fb49c90d,907f465458595b5b
Lung_Opacity-3795.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3795.png,Lung_Opacity/masks/Lung_Opacity-3795.png,True,G06508,train,5b595de26e353f202b0ffb90643a4c4c,807fff061055175d
Lung_Opacity-3796.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3796.png,Lung_Opacity/masks/Lung_Opacity-3796.png,True,G06509,train,73297c6a7b1c3948dc2223084e1e3290,c03d410a0f7d3f3d
Lung_Opacity-3797.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3797.png,Lung_Opacity/masks/Lung_Opacity-3797.png,True,G06510,train,3880f8a948c35663c0bdc7d0360fc466,903bc30f4f1b5d0e
Lung_Opacity-3798.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3798.png,Lung_Opacity/masks/Lung_Opacity-3798.png,True,G06511,train,985354357687b560da7294f02d3f5a2c,c83f170a1d587a5d
Lung_Opacity-3799.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3799.png,Lung_Opacity/masks/Lung_Opacity-3799.png,True,G06512,train,23bd41a438ce5562bc9ef7aff6ff6f7c,c17f464e1e9c8a39
Lung_Opacity-38.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-38.png,Lung_Opacity/masks/Lung_Opacity-38.png,True,G06513,train,7fb2c367b90f5b21b3a3f5603ac50abc,89785a5f4a3939aa
Lung_Opacity-380.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-380.png,Lung_Opacity/masks/Lung_Opacity-380.png,True,G06514,train,059a56d13eb7e36a56d463568ee3aed7,85b07e9b79c63063
Lung_Opacity-3800.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3800.png,Lung_Opacity/masks/Lung_Opacity-3800.png,True,G06515,train,3a4092146a5bc51db2a37f827f85f6a1,c54c0a1b4c7c6ded
Lung_Opacity-3801.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3801.png,Lung_Opacity/masks/Lung_Opacity-3801.png,True,G06516,train,e2d96729e4a7a1636d9365d3efc93a95,997f861e1f001d1f
Lung_Opacity-3802.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3802.png,Lung_Opacity/masks/Lung_Opacity-3802.png,True,G06517,train,c9164a7fd6b9c552a7f147180b519694,d42f0e0d1d5c2e5e
Lung_Opacity-3803.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3803.png,Lung_Opacity/masks/Lung_Opacity-3803.png,True,G06518,train,8639f4b6da000f606efbff453c3dc53a,dc2f424f191e4c6d
Lung_Opacity-3804.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3804.png,Lung_Opacity/masks/Lung_Opacity-3804.png,True,G06519,val,fa613ef73fc0963c54152f5be0becddc,d13b44471a591b5f
Lung_Opacity-3805.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3805.png,Lung_Opacity/masks/Lung_Opacity-3805.png,True,G06520,val,b4bd7bd8fc37926cd5e972e617fc1eac,c03d7dd30b0f0e2d
Lung_Opacity-3806.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3806.png,Lung_Opacity/masks/Lung_Opacity-3806.png,True,G06521,val,a3de72bbe6fdbf756a40b2c0f6abefc4,d46e434f2e4ca476
Lung_Opacity-3807.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3807.png,Lung_Opacity/masks/Lung_Opacity-3807.png,True,G06522,test,d097b138f37720ea0c25aa11713d52a5,d43f4b0c0f2d2d3c
Lung_Opacity-3808.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3808.png,Lung_Opacity/masks/Lung_Opacity-3808.png,True,G06523,test,1b3ad319ec4c4645812440f5020849a1,d831122c9e9697af
Lung_Opacity-3809.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3809.png,Lung_Opacity/masks/Lung_Opacity-3809.png,True,G06524,train,314fd48acbcd67f64604b64f34aa8501,d17d065e02475a5f
Lung_Opacity-381.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-381.png,Lung_Opacity/masks/Lung_Opacity-381.png,True,G06525,test,a2319f2911a577a26921d78fd640e93a,c46f014b0f0f2d6f
Lung_Opacity-3810.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3810.png,Lung_Opacity/masks/Lung_Opacity-3810.png,True,G06526,train,457cd61ba5137507a141722fc2ab7f2b,812b7a46965c595f
Lung_Opacity-3811.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3811.png,Lung_Opacity/masks/Lung_Opacity-3811.png,True,G06527,train,646a4bae40c9fb3b0cd20f0c3cb28121,817fc6465a575558
Lung_Opacity-3812.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3812.png,Lung_Opacity/masks/Lung_Opacity-3812.png,True,G06528,train,55087779cefccef70acab7e8bf83bccb,d02f83190d4d1e7f
Lung_Opacity-3813.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3813.png,Lung_Opacity/masks/Lung_Opacity-3813.png,True,G06529,train,a47e1778ca56dd94593874ab09f7799f,993947061e496d7e
Lung_Opacity-3814.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3814.png,Lung_Opacity/masks/Lung_Opacity-3814.png,True,G06530,train,a6085c1546d02e8a9dc552ae82ee2747,985a521a1c3b2faf
Lung_Opacity-3815.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3815.png,Lung_Opacity/masks/Lung_Opacity-3815.png,True,G06531,train,b63504b39d11208f904a3124dcd1aef1,cc27430e2e2ee4fc
Lung_Opacity-3816.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3816.png,Lung_Opacity/masks/Lung_Opacity-3816.png,True,G06532,train,7e8e8f24b0ce1938560bea3bd4b24f8b,817946861e5dea1f
Lung_Opacity-3817.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3817.png,Lung_Opacity/masks/Lung_Opacity-3817.png,True,G06533,train,672235f5cbc0a0fdf8494f09e636aeca,883c534d4b6c7c5e
Lung_Opacity-3818.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3818.png,Lung_Opacity/masks/Lung_Opacity-3818.png,True,G06534,train,4fb5fae5c4a8fb08c6efa3103532c6b2,817f6307131d3d1e
Lung_Opacity-3819.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3819.png,Lung_Opacity/masks/Lung_Opacity-3819.png,True,G06535,val,3bacb3e6b170c4870446513ea793e052,81790e5a185f793d
Lung_Opacity-382.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-382.png,Lung_Opacity/masks/Lung_Opacity-382.png,True,G06536,train,1b81ef7b784280050660831a08e32d54,d53a131b0a692bed
Lung_Opacity-3820.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3820.png,Lung_Opacity/masks/Lung_Opacity-3820.png,True,G06537,val,91787b5a3a2f37d7bddfd4df597a4ef8,d15a560a1f0aaf3d
Lung_Opacity-3821.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3821.png,Lung_Opacity/masks/Lung_Opacity-3821.png,True,G06538,train,a96c7d7185d81816a0c25094511058eb,97686c7a69273692
Lung_Opacity-3822.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3822.png,Lung_Opacity/masks/Lung_Opacity-3822.png,True,G06539,train,7abaa0b54396b2d41d5283fadb33a093,8f7e9497322aa8aa
Lung_Opacity-3823.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3823.png,Lung_Opacity/masks/Lung_Opacity-3823.png,True,G06540,train,6ae06200379ee1e733bc7e5a432d5367,817b85869e1a1b7b
Lung_Opacity-3824.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3824.png,Lung_Opacity/masks/Lung_Opacity-3824.png,True,G06541,val,12351d0b512bdedad2720a317bf93597,c938161d1d79b965
Lung_Opacity-3825.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3825.png,Lung_Opacity/masks/Lung_Opacity-3825.png,True,G06542,test,2a151e556f639d0dd97937837c5e09cb,993bd65a10595b1e
Lung_Opacity-3826.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3826.png,Lung_Opacity/masks/Lung_Opacity-3826.png,True,G06543,test,621031499f907b144c52463f2f79b67b,c86f461e1b4f4f0c
Lung_Opacity-3827.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3827.png,Lung_Opacity/masks/Lung_Opacity-3827.png,True,G06544,test,a67531deb50dd7c9f9dfd05afed41883,917a52161e5a5a3f
Lung_Opacity-3828.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3828.png,Lung_Opacity/masks/Lung_Opacity-3828.png,True,G06545,train,cf390745d5b8cde698664aefa68fd474,812bc49659797a5b
Lung_Opacity-3829.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3829.png,Lung_Opacity/masks/Lung_Opacity-3829.png,True,G06546,train,7e204648c88eea1ac4fe4b7d55110c1d,ca2d4d091d1f1f9a
Lung_Opacity-383.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-383.png,Lung_Opacity/masks/Lung_Opacity-383.png,True,G06547,train,7f39cf87fb01113bfce3abc1d13b6a41,d83b931f1432b3e1
Lung_Opacity-3830.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3830.png,Lung_Opacity/masks/Lung_Opacity-3830.png,True,G06548,train,6a244068997c1a831a2f9a25d82e1534,9c6a431f4c1f4a3e
Lung_Opacity-3831.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3831.png,Lung_Opacity/masks/Lung_Opacity-3831.png,True,G06549,train,669fdc47bec21b75861448795dbbc4b3,dc2ad34b0e5c293d
Lung_Opacity-3832.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3832.png,Lung_Opacity/masks/Lung_Opacity-3832.png,True,G06550,train,28f5bc41fb64a92bdd51bc1a53aa5c99,dc2d430e2cb4d5d5
Lung_Opacity-3833.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3833.png,Lung_Opacity/masks/Lung_Opacity-3833.png,True,G06551,train,bb260e050108d7a18c4e5cd76f2a3510,c17f564a1e5a584e
Lung_Opacity-3834.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3834.png,Lung_Opacity/masks/Lung_Opacity-3834.png,True,G06552,train,1712c3ff45a3f5494ab0afe64831528d,ddbcc20bd24b584b
Lung_Opacity-3835.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3835.png,Lung_Opacity/masks/Lung_Opacity-3835.png,True,G06553,train,c065ec883c4164072689fe6d526040c5,d42d6395914e6d35
Lung_Opacity-3836.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3836.png,Lung_Opacity/masks/Lung_Opacity-3836.png,True,G06554,train,4cda723c2213f733bb115cf91bc56c8e,fb788c115a0b1b5e
Lung_Opacity-3837.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3837.png,Lung_Opacity/masks/Lung_Opacity-3837.png,True,G06555,train,216645f5d1d28076d362735708a6b6e4,903f83129bd70b5d
Lung_Opacity-3838.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3838.png,Lung_Opacity/masks/Lung_Opacity-3838.png,True,G06556,train,6e2575e5175b513f8076b12c090fb785,d03c6bc28f1c2d6d
Lung_Opacity-3839.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3839.png,Lung_Opacity/masks/Lung_Opacity-3839.png,True,G06557,train,50dbbdde8ab9960aadc653fd2776dafa,f97f855e820b7901
Lung_Opacity-384.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-384.png,Lung_Opacity/masks/Lung_Opacity-384.png,True,G06558,train,f9bb3ce224a52b19696c3c764cb2cc5d,9879020f595b7b78
Lung_Opacity-3840.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3840.png,Lung_Opacity/masks/Lung_Opacity-3840.png,True,G06559,val,3fbb1358a04375b2a39eb66201258d07,91785ed69a1aa96a
Lung_Opacity-3841.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3841.png,Lung_Opacity/masks/Lung_Opacity-3841.png,True,G06560,val,32188079d65cc60c4d400dfb271d31ce,857e051e3a6e6a6a
Lung_Opacity-3842.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3842.png,Lung_Opacity/masks/Lung_Opacity-3842.png,True,G06561,test,cf925981e0c1e5bab060afcc6a233706,9378111b1b7b78e8
Lung_Opacity-3843.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3843.png,Lung_Opacity/masks/Lung_Opacity-3843.png,True,G06562,test,782e861fcbceb71432dee645de269974,9de0165b1c5a6b4e
Lung_Opacity-3844.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3844.png,Lung_Opacity/masks/Lung_Opacity-3844.png,True,G06563,train,7d8a804d9762914558a17f219df91ceb,d42cd3c3554ad34b
Lung_Opacity-3845.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3845.png,Lung_Opacity/masks/Lung_Opacity-3845.png,True,G06564,train,f5427a9ec331dbbfc06772f1238b77c3,8168477e79697526
Lung_Opacity-3846.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3846.png,Lung_Opacity/masks/Lung_Opacity-3846.png,True,G06565,train,83b735cc9f922bf7421f64eab05bcbd4,dd2815821e5d2f7c
Lung_Opacity-3847.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3847.png,Lung_Opacity/masks/Lung_Opacity-3847.png,True,G06566,val,8e710e0c0adce0210a2851254222a8c4,9d68431e4d0e0f3f
Lung_Opacity-3848.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3848.png,Lung_Opacity/masks/Lung_Opacity-3848.png,True,G06567,test,b3d77f50e6911948553be77ecfdc8f79,995b471e4d0d1e1b
Lung_Opacity-3849.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3849.png,Lung_Opacity/masks/Lung_Opacity-3849.png,True,G06568,val,8c10321a0f95f75c1b676cf5fc3ac692,d52f523d59456c19
Lung_Opacity-385.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-385.png,Lung_Opacity/masks/Lung_Opacity-385.png,True,G06569,test,dc61cbf6407841441a373ea28f3956c5,d4ae831b870f1d0f
Lung_Opacity-3850.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3850.png,Lung_Opacity/masks/Lung_Opacity-3850.png,True,G06570,val,c505e5b8e3b7fcbfbdbf7e3698a6fa40,817eafd618490f39
Lung_Opacity-3851.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3851.png,Lung_Opacity/masks/Lung_Opacity-3851.png,True,G06571,train,5f614a59088bcd32ea5f09e020fcb5d2,dc3cd3431a864a9f
Lung_Opacity-3852.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3852.png,Lung_Opacity/masks/Lung_Opacity-3852.png,True,G06572,train,47aec7729b41e785330997757683a85c,803972d5d614de57
Lung_Opacity-3853.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3853.png,Lung_Opacity/masks/Lung_Opacity-3853.png,True,G06573,train,8cf0aa5032ae02e1f521e11f81639dbd,887e961a585e58de
Lung_Opacity-3854.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3854.png,Lung_Opacity/masks/Lung_Opacity-3854.png,True,G06574,train,fe84f1275e2278aaeeb594ec65ebdc99,957dc2165a5a5a5a
Lung_Opacity-3855.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3855.png,Lung_Opacity/masks/Lung_Opacity-3855.png,True,G06575,test,9182e5e6c8ccb71246e86b0af49822fb,c42ad10f0e4f8b5f
Lung_Opacity-3856.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3856.png,Lung_Opacity/masks/Lung_Opacity-3856.png,True,G06576,train,2cfeb108e5bcd14fb47926cefe79d0f3,887f061a58fde8b8
Lung_Opacity-3857.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3857.png,Lung_Opacity/masks/Lung_Opacity-3857.png,True,G06577,val,0478bfd45e57148600054f75d935f9c7,8861c35e5e495b5b
Lung_Opacity-3858.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3858.png,Lung_Opacity/masks/Lung_Opacity-3858.png,True,G06578,train,3899c20769fa7d96c2b7f4d86f6e1f69,d52bd24d4f7c0c0e
Lung_Opacity-3859.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3859.png,Lung_Opacity/masks/Lung_Opacity-3859.png,True,G06579,train,9a8598cc59db8913a003518af1d249cc,d81b1618194f1f5f
Lung_Opacity-386.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-386.png,Lung_Opacity/masks/Lung_Opacity-386.png,True,G06580,train,7a1b3af29058a5fa1f6cbca742af06b7,cd3f52461a5a592d
Lung_Opacity-3860.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3860.png,Lung_Opacity/masks/Lung_Opacity-3860.png,True,G06581,train,b8134daf62f52422e09046d73f5de382,c93b535a584a5b3a
Lung_Opacity-3861.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3861.png,Lung_Opacity/masks/Lung_Opacity-3861.png,True,G06582,train,8d64432fb81dc3a5d3f73563b4267da8,d56b53434a185b6b
Lung_Opacity-3862.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3862.png,Lung_Opacity/masks/Lung_Opacity-3862.png,True,G06583,train,0cf5123f61cf8098b9a8244b9c8d9141,dc28410a1e3cbdf7
Lung_Opacity-3863.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3863.png,Lung_Opacity/masks/Lung_Opacity-3863.png,True,G06584,test,87b262b4617778e6ec599ee638aba9bd,d12b461e39296ece
Lung_Opacity-3864.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3864.png,Lung_Opacity/masks/Lung_Opacity-3864.png,True,G06585,train,c3843919fc4ffa35219d377e02e3c511,9170c65a5fce5a49
Lung_Opacity-3865.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3865.png,Lung_Opacity/masks/Lung_Opacity-3865.png,True,G06586,train,f29a58a4f02a9affca07e90376fd6540,9818165a5bfaeaaa
Lung_Opacity-3866.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3866.png,Lung_Opacity/masks/Lung_Opacity-3866.png,True,G06587,train,5029601236ed24d5982b067d1c818263,d6350f070f0ece0f
Lung_Opacity-3867.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3867.png,Lung_Opacity/masks/Lung_Opacity-3867.png,True,G06588,train,8a20f3374929bf7a54e2588d70b5f403,882d8616597d5e7a
Lung_Opacity-3868.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3868.png,Lung_Opacity/masks/Lung_Opacity-3868.png,True,G06589,train,2fd67f6d0424b915b60a5f46279e40c1,dc3c43c35f494b68
Lung_Opacity-3869.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3869.png,Lung_Opacity/masks/Lung_Opacity-3869.png,True,G06590,train,0d8e31ee9f311f29956ca0e1e8c198c6,c128125e585f5b5f
Lung_Opacity-387.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-387.png,Lung_Opacity/masks/Lung_Opacity-387.png,True,G06591,train,026300abee7294cadd74408abc5052e1,dc2d490faa14b5f2
Lung_Opacity-3870.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3870.png,Lung_Opacity/masks/Lung_Opacity-3870.png,True,G06592,train,1db19ae3e6739bb7c703bd46e4cf1148,c03d56025b5b5f59
Lung_Opacity-3871.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3871.png,Lung_Opacity/masks/Lung_Opacity-3871.png,True,G06593,train,3ff50d4d21851c4f45eef049f8c2b03f,8c6c871bd54f58c5
Lung_Opacity-3872.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3872.png,Lung_Opacity/masks/Lung_Opacity-3872.png,True,G06594,train,d3c048e3d0389e9e0af21a3962b3d459,d47d030f0a0f1b3f
Lung_Opacity-3873.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3873.png,Lung_Opacity/masks/Lung_Opacity-3873.png,True,G06595,train,d3cb0a6d1c3bdcb4e8f03f2d9f403ebb,dc25c30d4f1e0f4e
Lung_Opacity-3874.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3874.png,Lung_Opacity/masks/Lung_Opacity-3874.png,True,G06596,val,688cd66b03198fe086b0aa651cc3c2af,817f865e1f19494b
Lung_Opacity-3875.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3875.png,Lung_Opacity/masks/Lung_Opacity-3875.png,True,G06597,train,68a4329eb26ca0916e603426239ce6b0,c6774b084d0f0f6e
Lung_Opacity-3876.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3876.png,Lung_Opacity/masks/Lung_Opacity-3876.png,True,G06598,train,83778a53679ec7e073a05f3eb231c6e7,c03d3fc634c11b9b
Lung_Opacity-3877.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3877.png,Lung_Opacity/masks/Lung_Opacity-3877.png,True,G06599,val,ab34d2818af1e25b21c1a12acb3cf21d,917b55845a5a7b29
Lung_Opacity-3878.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3878.png,Lung_Opacity/masks/Lung_Opacity-3878.png,True,G06600,train,06b4a240be9e991df0b4f3536d02890e,803f5e5a595d584d
Lung_Opacity-3879.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3879.png,Lung_Opacity/masks/Lung_Opacity-3879.png,True,G06601,test,b8e3ee799e812de8810c6051b8a68fc2,d52a16c7430d5d5d
Lung_Opacity-388.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-388.png,Lung_Opacity/masks/Lung_Opacity-388.png,True,G06602,val,8439c1286bc05d424c52c55adbc3984b,d42d03135d4d0f5f
Lung_Opacity-3880.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3880.png,Lung_Opacity/masks/Lung_Opacity-3880.png,True,G06603,test,cff91acb0edf220998720b13e25545fd,887d4418597b6eb6
Lung_Opacity-3881.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3881.png,Lung_Opacity/masks/Lung_Opacity-3881.png,True,G06604,test,7b7353ebb7b0aac10de6891beeedad57,d42b515f4b4e2e2a
Lung_Opacity-3882.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3882.png,Lung_Opacity/masks/Lung_Opacity-3882.png,True,G06605,train,c442474466d81f781706a7c982fc9050,d47e21d30b0d8d8f
Lung_Opacity-3883.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3883.png,Lung_Opacity/masks/Lung_Opacity-3883.png,True,G06606,train,ace57d528edad7964269eaaad1a56f3d,d52f52514b0b4b1f
Lung_Opacity-3884.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3884.png,Lung_Opacity/masks/Lung_Opacity-3884.png,True,G06607,train,2d648518f69dfd5bd8d01acfab3c4ba6,c43fd30f454d1c4d
Lung_Opacity-3885.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3885.png,Lung_Opacity/masks/Lung_Opacity-3885.png,True,G06608,train,abe8ce095d558d2a09228f17b085c4a0,99698612974f1bd3
Lung_Opacity-3886.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3886.png,Lung_Opacity/masks/Lung_Opacity-3886.png,True,G06609,train,5bcde3809d29c728773f44bc45ff4e67,80787fa371964a9b
Lung_Opacity-3887.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3887.png,Lung_Opacity/masks/Lung_Opacity-3887.png,True,G06610,train,b8d345f36fb9895bc34f4e335e8fed9f,981b121a1afdf0b7
Lung_Opacity-3888.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3888.png,Lung_Opacity/masks/Lung_Opacity-3888.png,True,G06611,train,19c601e2c50aed67682906834e5349b1,d03d4b839e0a89fd
Lung_Opacity-3889.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3889.png,Lung_Opacity/masks/Lung_Opacity-3889.png,True,G06612,train,103fc7d690caa19dd99f433c2f957a35,c42fc54a6e60f1f1
Lung_Opacity-389.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-389.png,Lung_Opacity/masks/Lung_Opacity-389.png,True,G06613,train,a93bc1683a001fc6a680fa9cc1a2550f,dc2bd31c1c681f87
Lung_Opacity-3890.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3890.png,Lung_Opacity/masks/Lung_Opacity-3890.png,True,G06614,train,369db5b93befa45ff54f4fa8442285e9,987f461a594b393a
Lung_Opacity-3891.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3891.png,Lung_Opacity/masks/Lung_Opacity-3891.png,True,G06615,train,e2f1f12e87f7891f5d97079da559a5e2,9d281e180d57797d
Lung_Opacity-3892.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3892.png,Lung_Opacity/masks/Lung_Opacity-3892.png,True,G06616,train,ed52defc63efc8836f33b70b7cda85f9,d43c63c20b0faf2b
Lung_Opacity-3893.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3893.png,Lung_Opacity/masks/Lung_Opacity-3893.png,True,G06617,train,1a0098f51a9994dda4cdcfe8553ca5f2,d03bc6471b1c4f1b
Lung_Opacity-3894.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3894.png,Lung_Opacity/masks/Lung_Opacity-3894.png,True,G06618,train,09bec86166a8bea71dffd356734478f1,813956de5a1b7a29
Lung_Opacity-3895.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3895.png,Lung_Opacity/masks/Lung_Opacity-3895.png,True,G06619,val,087834d8984cb31ef64e1f088910f46a,c83f955a145a1b5d
Lung_Opacity-3896.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3896.png,Lung_Opacity/masks/Lung_Opacity-3896.png,True,G06620,val,dd98a11dbc01547bdb638753e094c391,d62741482d67276f
Lung_Opacity-3897.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3897.png,Lung_Opacity/masks/Lung_Opacity-3897.png,True,G06621,val,4baa54668dc8d39745a9a44b11bed3cf,95780e1a7a6e7a61
Lung_Opacity-3898.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3898.png,Lung_Opacity/masks/Lung_Opacity-3898.png,True,G06622,test,8a900842fcd7c3b401e2cae0d7a579e0,9d68c5975bc3055c
Lung_Opacity-3899.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3899.png,Lung_Opacity/masks/Lung_Opacity-3899.png,True,G06623,test,594494fe33fed59dabe9a27eeed486a2,81797945e48793ba
Lung_Opacity-39.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-39.png,Lung_Opacity/masks/Lung_Opacity-39.png,True,G06624,test,bdbc6cfc7f9256c6d9ed23a5c3b7b7a2,dc0bc32eaaa9aea4
Lung_Opacity-390.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-390.png,Lung_Opacity/masks/Lung_Opacity-390.png,True,G06625,train,206447810bd72a9e4b5fc5d72dd86833,847c474a0f4d6d7c
Lung_Opacity-3900.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3900.png,Lung_Opacity/masks/Lung_Opacity-3900.png,True,G06626,train,9583f2c8ac4a2b37333d0759b0c7409d,c47f03571b45071f
Lung_Opacity-3901.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3901.png,Lung_Opacity/masks/Lung_Opacity-3901.png,True,G06627,train,9692a96d2812ac8bec5493c9b11794cf,d42f555f0c1d0f0b
Lung_Opacity-3902.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3902.png,Lung_Opacity/masks/Lung_Opacity-3902.png,True,G06628,train,64fd06a5314faa90ef0d2da53c700ef6,d939d40a1d3b2d2d
Lung_Opacity-3903.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3903.png,Lung_Opacity/masks/Lung_Opacity-3903.png,True,G06629,train,fbea1753f563b46e0455f77033e3398e,996c0f1a4e794c79
Lung_Opacity-3904.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3904.png,Lung_Opacity/masks/Lung_Opacity-3904.png,True,G06630,val,37afcf8f3cfc6f379ec1e15a1038789a,9f700ed7921b511e
Lung_Opacity-3905.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3905.png,Lung_Opacity/masks/Lung_Opacity-3905.png,True,G06631,train,96ef5ff2ad614eb4911c570ac5b279ba,80395a0abdba9b9e
Lung_Opacity-3906.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3906.png,Lung_Opacity/masks/Lung_Opacity-3906.png,True,G06632,train,e34a2c30e6622c39e2628440b4eb506b,8c4e870d4c4eeadb
Lung_Opacity-3907.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3907.png,Lung_Opacity/masks/Lung_Opacity-3907.png,True,G06633,train,63486a6f6393a987b5bc48173201dc4a,817f671605171e5e
Lung_Opacity-3908.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3908.png,Lung_Opacity/masks/Lung_Opacity-3908.png,True,G06634,train,3638dcb6ece1b0c4ab2548f9265511c6,d129d6065a5b5b1b
Lung_Opacity-3909.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3909.png,Lung_Opacity/masks/Lung_Opacity-3909.png,True,G06635,train,541721cbdad2a4c6a2c311a94bf457f9,812fd678591b781e
Lung_Opacity-391.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-391.png,Lung_Opacity/masks/Lung_Opacity-391.png,True,G06636,train,c9229ac808eb86bcf7d131cb1f980e77,c035515b4a4b6f6e
Lung_Opacity-3910.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3910.png,Lung_Opacity/masks/Lung_Opacity-3910.png,True,G06637,train,a7141a7c7deccc6f7d88ceef6b16800f,c86f95b05317564b
Lung_Opacity-3911.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3911.png,Lung_Opacity/masks/Lung_Opacity-3911.png,True,G06638,train,7383a011740e49d907f70c1fd8c4d75a,9139eb4596165f0e
Lung_Opacity-3912.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3912.png,Lung_Opacity/masks/Lung_Opacity-3912.png,True,G06639,train,ca9e48edf498181d6865c18c588ae8f1,917a961a1b3f94a6
Lung_Opacity-3913.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3913.png,Lung_Opacity/masks/Lung_Opacity-3913.png,True,G06640,val,81b6b89da0466728bc14be4256bfa41c,dd0c071b5e7a7868
Lung_Opacity-3914.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3914.png,Lung_Opacity/masks/Lung_Opacity-3914.png,True,G06641,val,642abfc228186212d74dc444c3214a3e,897d045e587969e9
Lung_Opacity-3915.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3915.png,Lung_Opacity/masks/Lung_Opacity-3915.png,True,G06642,test,af66b3eb1043a2af6bda0e4ef97b59a6,c168fa4b5d1e190f
Lung_Opacity-3916.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3916.png,Lung_Opacity/masks/Lung_Opacity-3916.png,True,G06643,test,fd0c5f91a687f11fe70a5ed3475e3ff6,d02dff4c1c487c3c
Lung_Opacity-3917.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3917.png,Lung_Opacity/masks/Lung_Opacity-3917.png,True,G06644,train,62fd7fe09ba151a30abdb5632bebb948,8179861e1a49ee7e
Lung_Opacity-3918.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3918.png,Lung_Opacity/masks/Lung_Opacity-3918.png,True,G06645,test,f8b1d8bf96dfc83954ad17f392459237,9069421e5f7b6f48
Lung_Opacity-3919.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3919.png,Lung_Opacity/masks/Lung_Opacity-3919.png,True,G06646,train,006541cfa8244e937b58bcf7f467006b,c02f430a5f5f5d64
Lung_Opacity-392.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-392.png,Lung_Opacity/masks/Lung_Opacity-392.png,True,G06647,train,b6bf49eba2aef58a4be0e7a7e93f28dc,813714177f3b7b20
Lung_Opacity-3920.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3920.png,Lung_Opacity/masks/Lung_Opacity-3920.png,True,G06648,train,8f462de856d483a6d812871127f5159e,817f8a18797b3e14
Lung_Opacity-3921.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3921.png,Lung_Opacity/masks/Lung_Opacity-3921.png,True,G06649,train,21e6b28ed85f8b837447f2e68ee29a9e,c83d14de1cc99eb8
Lung_Opacity-3922.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3922.png,Lung_Opacity/masks/Lung_Opacity-3922.png,True,G06650,train,64e7c511f82565124d31a12a0fe3fa36,917ae686184d5b5d
Lung_Opacity-3923.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3923.png,Lung_Opacity/masks/Lung_Opacity-3923.png,True,G06651,train,d3896b1078f3a4d1c4dae995641ddfa8,807b0d1d535f7a52
Lung_Opacity-3924.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3924.png,Lung_Opacity/masks/Lung_Opacity-3924.png,True,G06652,train,9704ac5c03381fd4200203efeab9e6c3,81689e1f1c7c396d
Lung_Opacity-3925.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3925.png,Lung_Opacity/masks/Lung_Opacity-3925.png,True,G06653,train,6c35e1f52f67838baa574cfcb68a2d70,903f52189eb6b4f4
Lung_Opacity-3926.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3926.png,Lung_Opacity/masks/Lung_Opacity-3926.png,True,G06654,train,3788af8a61f19a5900d832e44efcdcdf,c968d6521a4f5b1b
Lung_Opacity-3927.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3927.png,Lung_Opacity/masks/Lung_Opacity-3927.png,True,G06655,train,8d2ec2a3abb2562940a6a773e4747d47,817906861f5bcecd
Lung_Opacity-3928.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3928.png,Lung_Opacity/masks/Lung_Opacity-3928.png,True,G06656,val,2520a1b49d5597ec6a810fdc8a0cc389,85ec6e877ae0b470
Lung_Opacity-3929.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3929.png,Lung_Opacity/masks/Lung_Opacity-3929.png,True,G06657,val,03f5a5896da0f47c8c7b483b9d099268,c83d4b1e2cb9bcb0
Lung_Opacity-393.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-393.png,Lung_Opacity/masks/Lung_Opacity-393.png,True,G06658,test,dda63c5ab27c89bcee04885c5d05316d,887a45971eca471f
Lung_Opacity-3930.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3930.png,Lung_Opacity/masks/Lung_Opacity-3930.png,True,G06659,train,61e104fbc5984a7a6ed8d67a6c3526b6,cd2d165253494b5f
Lung_Opacity-3931.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3931.png,Lung_Opacity/masks/Lung_Opacity-3931.png,True,G06660,train,f985113f6fbed6f44b715b47da5ab640,dd68d34f5c0b4c1a
Lung_Opacity-3932.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3932.png,Lung_Opacity/masks/Lung_Opacity-3932.png,True,G06661,train,e5960cb0ae9a61945800cc3234312f2e,917b02165e1a6b7b
Lung_Opacity-3933.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3933.png,Lung_Opacity/masks/Lung_Opacity-3933.png,True,G06662,train,41be9152bce35dda330bb2b00c88675f,d83d435b5f48682e
Lung_Opacity-3934.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3934.png,Lung_Opacity/masks/Lung_Opacity-3934.png,True,G06663,train,989d03402e73a80de1d05029d79ef759,816be4145f595d55
Lung_Opacity-3935.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3935.png,Lung_Opacity/masks/Lung_Opacity-3935.png,True,G06664,train,a9e0fb3b35f67a71fb00c2d57adafa15,9d603c563a6c6c6e
Lung_Opacity-3936.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3936.png,Lung_Opacity/masks/Lung_Opacity-3936.png,True,G06665,val,7d3c4cd4f03cbc2c89733be26e4900e9,812856575c9d6bcb
Lung_Opacity-3937.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3937.png,Lung_Opacity/masks/Lung_Opacity-3937.png,True,G06666,train,84c627775804255edeaf2504b156ac71,df681f1e5b585828
Lung_Opacity-3938.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3938.png,Lung_Opacity/masks/Lung_Opacity-3938.png,True,G06667,val,7d822e86b4d7d84ec8fa20e58eb67ce3,d83dc247194b2f3c
Lung_Opacity-3939.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3939.png,Lung_Opacity/masks/Lung_Opacity-3939.png,True,G06668,test,da7b13a6c4143415d4cbdbbe3c964e98,d52a530b0f2d2d2f
Lung_Opacity-394.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-394.png,Lung_Opacity/masks/Lung_Opacity-394.png,True,G06669,test,88a42239e9b7215695fb7f49f0d76a7d,914f865e5a6f6a60
Lung_Opacity-3940.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3940.png,Lung_Opacity/masks/Lung_Opacity-3940.png,True,G06670,train,31b51be1f337f8c8085c64f05545ba6a,c03d421f5979395d
Lung_Opacity-3941.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3941.png,Lung_Opacity/masks/Lung_Opacity-3941.png,True,G06671,train,ea83f479a5d76b847fdf0818d28f8201,dd3a131b0d185b1f
Lung_Opacity-3942.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3942.png,Lung_Opacity/masks/Lung_Opacity-3942.png,True,G06672,train,5e0ac1bce8956e855e1e746fc530d405,cc7e53031b6c5a1b
Lung_Opacity-3943.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3943.png,Lung_Opacity/masks/Lung_Opacity-3943.png,True,G06673,train,bca299286b25579d8a487a82d1a2f65c,987dc65a1958495f
Lung_Opacity-3944.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3944.png,Lung_Opacity/masks/Lung_Opacity-3944.png,True,G06674,train,e7ec3068bc639d375597e46856f68863,913986161a7d7f38
Lung_Opacity-3945.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3945.png,Lung_Opacity/masks/Lung_Opacity-3945.png,True,G06675,train,ce9f0e2b5e7c6b578eb01dd0305b3876,d42f530a2e333573
Lung_Opacity-3946.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3946.png,Lung_Opacity/masks/Lung_Opacity-3946.png,True,G06676,train,22b028a92cebfe73bddb9e5e2ddac16c,c17b501f19293d3d
Lung_Opacity-3947.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3947.png,Lung_Opacity/masks/Lung_Opacity-3947.png,True,G06677,train,aa3c4eb079ab57575bf2f612c112796b,95688f1e59496c79
Lung_Opacity-3948.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3948.png,Lung_Opacity/masks/Lung_Opacity-3948.png,True,G06678,train,a31d3e37e2cd7e3a9723811fc2079d16,d52806de3b69e436
Lung_Opacity-3949.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3949.png,Lung_Opacity/masks/Lung_Opacity-3949.png,True,G06679,val,c4df52112d46a6e026a041755676a467,d42b520b0cecbdbc
Lung_Opacity-395.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-395.png,Lung_Opacity/masks/Lung_Opacity-395.png,True,G06680,train,29ac4764a3862390b076eacafada1963,d52a53434f4e5e4a
Lung_Opacity-3950.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3950.png,Lung_Opacity/masks/Lung_Opacity-3950.png,True,G06681,test,6a44400ce8a6f64d8de0a41f711ff158,d42e85d70d4c6e74
Lung_Opacity-3951.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3951.png,Lung_Opacity/masks/Lung_Opacity-3951.png,True,G06682,val,56b4807a395c82d87c8faeb56698067d,d03b5a2d34756666
Lung_Opacity-3952.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3952.png,Lung_Opacity/masks/Lung_Opacity-3952.png,True,G06683,test,639918730b9505f25a9972fe4f49efa7,81395e865a3be94b
Lung_Opacity-3953.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3953.png,Lung_Opacity/masks/Lung_Opacity-3953.png,True,G06684,test,ce6da3ceb77ba6891d7ad7ebeb4d4cf9,8d2d425f4d6e6872
Lung_Opacity-3954.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3954.png,Lung_Opacity/masks/Lung_Opacity-3954.png,True,G06685,train,05ad5b62f550e3f5ef348f705aeadc37,cc3ed3534d4a683a
Lung_Opacity-3955.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3955.png,Lung_Opacity/masks/Lung_Opacity-3955.png,True,G06686,train,f3f968fb06085fa13eac25ef931e9839,dc2c821b1bc71f69
Lung_Opacity-3956.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3956.png,Lung_Opacity/masks/Lung_Opacity-3956.png,True,G06687,train,f3cdb81d1929ab6fafc09503c3a22536,996886175a4e5b79
Lung_Opacity-3957.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3957.png,Lung_Opacity/masks/Lung_Opacity-3957.png,True,G06688,train,01c2bdda29d4415412586d22f898227f,9d6ec21a5deca8b0
Lung_Opacity-3958.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3958.png,Lung_Opacity/masks/Lung_Opacity-3958.png,True,G06689,train,ccab77279fe979d65067c28a55956500,d46c0d0e4a7dc3d3
Lung_Opacity-3959.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3959.png,Lung_Opacity/masks/Lung_Opacity-3959.png,True,G06690,train,f7e69c56698f73a4e56f34dcd9933548,991e163979e5a1e1
Lung_Opacity-396.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-396.png,Lung_Opacity/masks/Lung_Opacity-396.png,True,G06691,train,15dba348d9071388363cfef895b8bae0,c13d52d2cb464b4f
Lung_Opacity-3960.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3960.png,Lung_Opacity/masks/Lung_Opacity-3960.png,True,G06692,train,0f77aba53a418cb93a2524df0403dd2b,d828d21697c3571f
Lung_Opacity-3961.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3961.png,Lung_Opacity/masks/Lung_Opacity-3961.png,True,G06693,train,b5253abb3dd3fbd9886cc619b150f3be,c83c03035d1eff0f
Lung_Opacity-3962.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3962.png,Lung_Opacity/masks/Lung_Opacity-3962.png,True,G06694,train,581e737ee870a5600024b19872e66545,dd2ec313094d3c7c
Lung_Opacity-3963.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3963.png,Lung_Opacity/masks/Lung_Opacity-3963.png,True,G06695,test,89d1fd48afda3ff85725a306c2cd6b44,c921d2497b7329b3
Lung_Opacity-3964.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3964.png,Lung_Opacity/masks/Lung_Opacity-3964.png,True,G06696,train,0e3d01fc5bb453f65f4978fad3894141,803dfa142ba1abaf
Lung_Opacity-3965.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3965.png,Lung_Opacity/masks/Lung_Opacity-3965.png,True,G06697,train,5a11afba2d87c459a19138c8fb2e5bf6,d03b571a595b3d28
Lung_Opacity-3966.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3966.png,Lung_Opacity/masks/Lung_Opacity-3966.png,True,G06698,train,11b6ffb7f190d304aafa19b2b423411c,cc6a474b0a2ca7b7
Lung_Opacity-3967.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3967.png,Lung_Opacity/masks/Lung_Opacity-3967.png,True,G06699,train,aee92385736e804ad7d13ecb814e6c8a,dd2f42135b012f3d
Lung_Opacity-3968.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3968.png,Lung_Opacity/masks/Lung_Opacity-3968.png,True,G06700,train,06a38423c662ebd38e44ec8ce749dd63,807b541c1a5b7b3d
Lung_Opacity-3969.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3969.png,Lung_Opacity/masks/Lung_Opacity-3969.png,True,G06701,val,e787a4c68d7238c43375719603d78946,81798d5c5a5b5b19
Lung_Opacity-397.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-397.png,Lung_Opacity/masks/Lung_Opacity-397.png,True,G06702,val,da763395015a54d7f69f42fd166823d8,c93c46181aefb8b5
Lung_Opacity-3970.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3970.png,Lung_Opacity/masks/Lung_Opacity-3970.png,True,G06703,val,955a51b56737a4b4a2e366530df428d0,bf589d72c0e3c4c4
Lung_Opacity-3971.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3971.png,Lung_Opacity/masks/Lung_Opacity-3971.png,True,G06704,test,328a906362998aca7ee087afb92ae2a3,8869d31a495b796d
Lung_Opacity-3972.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3972.png,Lung_Opacity/masks/Lung_Opacity-3972.png,True,G06705,test,b6f2a5cc4e308e524635a0959fa089f0,896f061a0a597f6d
Lung_Opacity-3973.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3973.png,Lung_Opacity/masks/Lung_Opacity-3973.png,True,G06706,train,50a50f50f9b68131599000987976922d,d028c7979f05df18
Lung_Opacity-3974.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3974.png,Lung_Opacity/masks/Lung_Opacity-3974.png,True,G06707,train,54748c44ea46b9b035145926873a6707,897856071b5de8ba
Lung_Opacity-3975.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3975.png,Lung_Opacity/masks/Lung_Opacity-3975.png,True,G06708,train,b6e607e474497f898fdf433eac718417,942f07071f1f1e5c
Lung_Opacity-3976.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3976.png,Lung_Opacity/masks/Lung_Opacity-3976.png,True,G06709,train,9573e826da12122f7a65b89f60ab6482,dd6cc0d31f0f490e
Lung_Opacity-3977.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3977.png,Lung_Opacity/masks/Lung_Opacity-3977.png,True,G06710,train,2bbe61dd2372a468ca77008cc20f5337,93301b1f3a1b3b59
Lung_Opacity-3978.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3978.png,Lung_Opacity/masks/Lung_Opacity-3978.png,True,G06711,train,0091abed4c0809d2be9bb7392cebdc60,dc2f431d0a2d1eb5
Lung_Opacity-3979.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3979.png,Lung_Opacity/masks/Lung_Opacity-3979.png,True,G06712,train,24040016718892d2cdf5f14f58d4ee86,8178475f1abce8a9
Lung_Opacity-398.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-398.png,Lung_Opacity/masks/Lung_Opacity-398.png,True,G06713,val,52457e93522ae766e27832ebf3fdf1a3,dd0d0a1c5d5aead2
Lung_Opacity-3980.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3980.png,Lung_Opacity/masks/Lung_Opacity-3980.png,True,G06714,train,c54e63f7e315be5dfcea4ab1ffd960d3,d42dc34a2d6a7273
Lung_Opacity-3981.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3981.png,Lung_Opacity/masks/Lung_Opacity-3981.png,True,G06715,train,75a6761ea8df8a3521a890d80b02b383,9c1cc34b1d683dbc
Lung_Opacity-3982.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3982.png,Lung_Opacity/masks/Lung_Opacity-3982.png,True,G06716,val,2f5df9b17c1f903238f2bb6a5e47a817,896816165e7d7a78
Lung_Opacity-3983.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3983.png,Lung_Opacity/masks/Lung_Opacity-3983.png,True,G06717,train,9d8d600a65bfa59d6e4f2784fabb88ce,99289e5a5a6c786b
Lung_Opacity-3984.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3984.png,Lung_Opacity/masks/Lung_Opacity-3984.png,True,G06718,test,b381bfd0f6838d06c76ba935977db762,9f601e16185b5b79
Lung_Opacity-3985.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3985.png,Lung_Opacity/masks/Lung_Opacity-3985.png,True,G06719,train,a7bab61c0a4a34be43ea4b719f528a97,dd2a43871e45ee1a
Lung_Opacity-3986.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3986.png,Lung_Opacity/masks/Lung_Opacity-3986.png,True,G06720,train,c4b322fb28ba426f385ed3a664987986,912f46161e4e3e5e
Lung_Opacity-3987.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3987.png,Lung_Opacity/masks/Lung_Opacity-3987.png,True,G06721,train,8a2eb9e674e618428029cd76f01a3999,95680fc79f1e0a1d
Lung_Opacity-3988.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3988.png,Lung_Opacity/masks/Lung_Opacity-3988.png,True,G06722,val,8c74d3d7bfe2df65e79257d2f498419f,816f865a796d68ac
Lung_Opacity-3989.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3989.png,Lung_Opacity/masks/Lung_Opacity-3989.png,True,G06723,test,16dea5eb181144aacd8bc03747c56ce6,d07bd7561c405a5b
Lung_Opacity-399.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-399.png,Lung_Opacity/masks/Lung_Opacity-399.png,True,G06724,test,d127bd4d1ec92f26c281e1aff45024c2,917fc61a194d583d
Lung_Opacity-3990.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3990.png,Lung_Opacity/masks/Lung_Opacity-3990.png,True,G06725,train,fbe161174a5029d6cbaff47260451094,d839d35f4d4d481a
Lung_Opacity-3991.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3991.png,Lung_Opacity/masks/Lung_Opacity-3991.png,True,G06726,train,11bbf0fb820c0cd39f9134a7139c88b1,d060474a5f4acf7a
Lung_Opacity-3992.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3992.png,Lung_Opacity/masks/Lung_Opacity-3992.png,True,G06727,train,4779456f4f46b5e6830d01b72dc0bea4,d42a030d6e2e2f6f
Lung_Opacity-3993.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3993.png,Lung_Opacity/masks/Lung_Opacity-3993.png,True,G06728,val,9f78666224f30349847c75884d423842,cc3ed3430b6e2f0c
Lung_Opacity-3994.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3994.png,Lung_Opacity/masks/Lung_Opacity-3994.png,True,G06729,test,d824286cacacbdff53f85136b1963039,817d065e5b1a1f39
Lung_Opacity-3995.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3995.png,Lung_Opacity/masks/Lung_Opacity-3995.png,True,G06730,train,82b7175783731245551facb7b7883c18,897f145a597968e1
Lung_Opacity-3996.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3996.png,Lung_Opacity/masks/Lung_Opacity-3996.png,True,G06731,train,37881f41098748a96358fc6d9b577959,d43d41435b0b6d6d
Lung_Opacity-3997.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3997.png,Lung_Opacity/masks/Lung_Opacity-3997.png,True,G06732,train,682efa86bae1c0c0bffa09b23ce33406,c17f141a5a1f195b
Lung_Opacity-3998.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3998.png,Lung_Opacity/masks/Lung_Opacity-3998.png,True,G06733,val,d85cc14291bf125d9ceb5cc9694c427d,d52a474f1e0e3c1e
Lung_Opacity-3999.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-3999.png,Lung_Opacity/masks/Lung_Opacity-3999.png,True,G06734,train,cbccd2c0f93e04ecc2cfc2c708aec6ea,dc3f421b5f420b59
Lung_Opacity-4.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4.png,Lung_Opacity/masks/Lung_Opacity-4.png,True,G06735,train,9d608876a34c60f4555e78c77c815913,d52a171f5d1e1c1c
Lung_Opacity-40.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-40.png,Lung_Opacity/masks/Lung_Opacity-40.png,True,G06736,test,e08baef8dccbe996eb280bf4d9d0d7b7,c62d59482e2f2f2e
Lung_Opacity-400.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-400.png,Lung_Opacity/masks/Lung_Opacity-400.png,True,G06737,train,b6b036192ef599051d1ef0c032581f0c,c96916c71e1c4e5e
Lung_Opacity-4000.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4000.png,Lung_Opacity/masks/Lung_Opacity-4000.png,True,G06738,train,8941ce63c8cdc5e4eb6dce3e2d191fa9,d42dc3074b0b4f6b
Lung_Opacity-4001.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4001.png,Lung_Opacity/masks/Lung_Opacity-4001.png,True,G06739,train,a7f594895bf37327c2077070d1e5339d,d52ec10f0f2e3d0b
Lung_Opacity-4002.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4002.png,Lung_Opacity/masks/Lung_Opacity-4002.png,True,G06740,train,ef45c8c69eeabd8c904f7d0bbb191aec,d42d471f185d6d1c
Lung_Opacity-4003.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4003.png,Lung_Opacity/masks/Lung_Opacity-4003.png,True,G06741,train,3454c3391dfc8e96c07caec691d5b76f,817816185b5f1b7b
Lung_Opacity-4004.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4004.png,Lung_Opacity/masks/Lung_Opacity-4004.png,True,G06742,val,0947685f2526a15c49898b3242c5c66c,917a0959b6b6a6d2
Lung_Opacity-4005.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4005.png,Lung_Opacity/masks/Lung_Opacity-4005.png,True,G06743,train,652c837b848a04ea1fee2e33f1309f3d,912a475e4a1f196f
Lung_Opacity-4006.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4006.png,Lung_Opacity/masks/Lung_Opacity-4006.png,True,G06744,test,2b2c21dea19aef56086226c5efc2538b,857b61585a7a7968
Lung_Opacity-4007.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4007.png,Lung_Opacity/masks/Lung_Opacity-4007.png,True,G06745,train,553719c124bf2e58ab28512028cc88b5,d52e071f095c4e7a
Lung_Opacity-4008.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4008.png,Lung_Opacity/masks/Lung_Opacity-4008.png,True,G06746,train,5a94372b89d6872746344e1978267f58,dd2d431f4d4b1c4a
Lung_Opacity-4009.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4009.png,Lung_Opacity/masks/Lung_Opacity-4009.png,True,G06747,train,24253a2765c5673d168d07ab0d902ae9,81bae5c47c947a95
Lung_Opacity-401.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-401.png,Lung_Opacity/masks/Lung_Opacity-401.png,True,G06748,train,8457059672a132e7a91707b1dee11bc0,d961561f461d471b
Lung_Opacity-4010.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4010.png,Lung_Opacity/masks/Lung_Opacity-4010.png,True,G06749,train,f66bf708c20ca8b8ea46c01fde597884,997985161f1e1e1b
Lung_Opacity-4011.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4011.png,Lung_Opacity/masks/Lung_Opacity-4011.png,True,G06750,train,c41d9d2eec99fdbcf7af3cdfcb859718,dd2fd20b4e4b191c
Lung_Opacity-4012.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4012.png,Lung_Opacity/masks/Lung_Opacity-4012.png,True,G06751,train,04ed1c5be9b59320855cc68432b176a0,de3d4a486a2e2e2e
Lung_Opacity-4013.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4013.png,Lung_Opacity/masks/Lung_Opacity-4013.png,True,G06752,val,3f68713daa963fc21ebfff2c0b0e9239,917920c41f5b5b5b
Lung_Opacity-4014.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4014.png,Lung_Opacity/masks/Lung_Opacity-4014.png,True,G06753,val,d75d3ade6f84e6d8b94419a1d629136b,d42b651f4c2c6c4f
Lung_Opacity-4015.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4015.png,Lung_Opacity/masks/Lung_Opacity-4015.png,True,G06754,train,ee5446c6aa07dc3d01c455d4df94d3b0,95690e06195edf78
Lung_Opacity-4016.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4016.png,Lung_Opacity/masks/Lung_Opacity-4016.png,True,G06755,val,1f74512d8b168b3cacde8ef703d7c0f5,d42cc30b4e4f3f2c
Lung_Opacity-4017.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4017.png,Lung_Opacity/masks/Lung_Opacity-4017.png,True,G06756,test,bd816c7a5d99eaaa0cc3835c59aae338,897986961a1f6d1d
Lung_Opacity-4018.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4018.png,Lung_Opacity/masks/Lung_Opacity-4018.png,True,G06757,test,18b05b4acc4cbd575444fc7e70cbe850,d14e061e0a5f3aeb
Lung_Opacity-4019.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4019.png,Lung_Opacity/masks/Lung_Opacity-4019.png,True,G06758,train,effa41c6fb90d48b9d337d1c5dae1acd,887753f9a09e4175
Lung_Opacity-402.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-402.png,Lung_Opacity/masks/Lung_Opacity-402.png,True,G06759,train,c7173787b8b42ccdda3e8c38001be000,91791e6aa1a6b656
Lung_Opacity-4020.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4020.png,Lung_Opacity/masks/Lung_Opacity-4020.png,True,G06760,train,e3c99fac12575df620f893a6ea1c2b9d,c52e145aca4ecf4b
Lung_Opacity-4021.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4021.png,Lung_Opacity/masks/Lung_Opacity-4021.png,True,G06761,train,9601d78a2648af9e882ea52525c5515f,913bd6171929a8bd
Lung_Opacity-4022.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4022.png,Lung_Opacity/masks/Lung_Opacity-4022.png,True,G06762,test,d106632897156d4598088615f4d70bad,c50c074f5b6c4dea
Lung_Opacity-4023.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4023.png,Lung_Opacity/masks/Lung_Opacity-4023.png,True,G06763,train,8fbd91bd91e20c045f5c35615349333d,915a961a3b35a7ac
Lung_Opacity-4024.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4024.png,Lung_Opacity/masks/Lung_Opacity-4024.png,True,G06764,train,6715ead89c36c4f7cc47c2d5839ab348,dd2ed2030f4d981f
Lung_Opacity-4025.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4025.png,Lung_Opacity/masks/Lung_Opacity-4025.png,True,G06765,train,15ccc33562cf15e5c42f3e1eaf64ad37,dc09830f2cbcbcb5
Lung_Opacity-4026.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4026.png,Lung_Opacity/masks/Lung_Opacity-4026.png,True,G06766,train,9d2ecf181122bef6aaa9990448566f53,9528075b5a4b3b4f
Lung_Opacity-4027.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4027.png,Lung_Opacity/masks/Lung_Opacity-4027.png,True,G06767,train,60c1f8fe35e3d12c60b1b920c6c454c8,dd2e50d30bd54a4b
Lung_Opacity-4028.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4028.png,Lung_Opacity/masks/Lung_Opacity-4028.png,True,G06768,train,bf43cbb9b49e1bd6bb96352b63de59e5,9d680f5a59797868
Lung_Opacity-4029.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4029.png,Lung_Opacity/masks/Lung_Opacity-4029.png,True,G06769,train,e0a1ac0069f9d4bef1d2f5301efe29f8,d52c055f0b4f4e5c
Lung_Opacity-403.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-403.png,Lung_Opacity/masks/Lung_Opacity-403.png,True,G06770,train,68f2656767857cb3c274e27a9dc5eadf,dd2fc0520aadea2e
Lung_Opacity-4030.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4030.png,Lung_Opacity/masks/Lung_Opacity-4030.png,True,G06771,train,2a863d5faae55f9b32e69aa1d699847c,8c2e861f0d0d7f47
Lung_Opacity-4031.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4031.png,Lung_Opacity/masks/Lung_Opacity-4031.png,True,G06772,train,513177c82d3f4da0e4ec6af6c7ba8afd,802e175e4d5e6c7c
Lung_Opacity-4032.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4032.png,Lung_Opacity/masks/Lung_Opacity-4032.png,True,G06773,train,3aadf532b92c46cad36b1e111c8f7b4e,dd39b6961e1e0a0b
Lung_Opacity-4033.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4033.png,Lung_Opacity/masks/Lung_Opacity-4033.png,True,G06774,train,cb719a1d07a3fa720b76d15c726496c1,d83cd1534b4b4a6e
Lung_Opacity-4034.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4034.png,Lung_Opacity/masks/Lung_Opacity-4034.png,True,G06775,train,e9d085e38b0a67ac86dfeeb4640472a9,987b061e181b5f1f
Lung_Opacity-4035.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4035.png,Lung_Opacity/masks/Lung_Opacity-4035.png,True,G06776,train,22a14fd3106f3fb16a3e428e552cd633,9178c54b29797978
Lung_Opacity-4036.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4036.png,Lung_Opacity/masks/Lung_Opacity-4036.png,True,G06777,train,4df7ddbca3e223af2f154e076efbaa30,c96b171a555e185d
Lung_Opacity-4037.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4037.png,Lung_Opacity/masks/Lung_Opacity-4037.png,True,G06778,train,9df58541ae3fb2c917daae378fb2bf1b,d43c07431a0c6fef
Lung_Opacity-4038.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4038.png,Lung_Opacity/masks/Lung_Opacity-4038.png,True,G06779,val,9473713872a7cc0f173857336d9c6b9f,d83f5308491f594f
Lung_Opacity-4039.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4039.png,Lung_Opacity/masks/Lung_Opacity-4039.png,True,G06780,val,c5a893f387dae2015fee6cff371e018b,917bc61859c93bc9
Lung_Opacity-404.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-404.png,Lung_Opacity/masks/Lung_Opacity-404.png,True,G06781,val,174635ff2619f3780a08bf41b430fea9,8b505e1b1b3939b9
Lung_Opacity-4040.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4040.png,Lung_Opacity/masks/Lung_Opacity-4040.png,True,G06782,test,8e971af13afd005c6d86467794956589,dd2ad30f540f540f
Lung_Opacity-4041.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4041.png,Lung_Opacity/masks/Lung_Opacity-4041.png,True,G06783,test,6eff9fcb2dac780c60240c95c274a198,dd2ed00b0f682f0f
Lung_Opacity-4042.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4042.png,Lung_Opacity/masks/Lung_Opacity-4042.png,True,G06784,test,0b4f6bc5dfd3d87d95d84931af306576,dd2ad34f160b1d4a
Lung_Opacity-4043.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4043.png,Lung_Opacity/masks/Lung_Opacity-4043.png,True,G06785,train,6aba1d0ea3a4a22c3e66f37a50ec87a4,817f06061a1b7b3f
Lung_Opacity-4044.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4044.png,Lung_Opacity/masks/Lung_Opacity-4044.png,True,G06786,train,c905bb0faf09cd250587285bfed31d85,986d871a594d5d59
Lung_Opacity-4045.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4045.png,Lung_Opacity/masks/Lung_Opacity-4045.png,True,G06787,train,7178b8eea8e5fe0d23b57899305a90db,8c6f861b497d446d
Lung_Opacity-4046.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4046.png,Lung_Opacity/masks/Lung_Opacity-4046.png,True,G06788,train,f33b2d95a45fdf5cd88b24f7720a1b82,d43d420b0b4f6f3c
Lung_Opacity-4047.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4047.png,Lung_Opacity/masks/Lung_Opacity-4047.png,True,G06789,train,7851d6cbfe3e54930616c905edeafc58,dc28d94b0e3c1e1f
Lung_Opacity-4048.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4048.png,Lung_Opacity/masks/Lung_Opacity-4048.png,True,G06790,val,5e461a5b7141a437c0fe5855e8cbd78f,d03f1a1a3eb583a5
Lung_Opacity-4049.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4049.png,Lung_Opacity/masks/Lung_Opacity-4049.png,True,G06791,val,90da3a7ee506acdd5c510d78cd6d8126,9465431b0f4f0f4f
Lung_Opacity-405.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-405.png,Lung_Opacity/masks/Lung_Opacity-405.png,True,G06792,train,9ec731e666efadbd162911699de6d865,997bc6161e501b5e
Lung_Opacity-4050.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4050.png,Lung_Opacity/masks/Lung_Opacity-4050.png,True,G06793,test,df2ea5962c60d4408099c70c126140d5,953ac2821b5dfd98
Lung_Opacity-4051.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4051.png,Lung_Opacity/masks/Lung_Opacity-4051.png,True,G06794,test,e54350d60b91d529236841e704823e40,dc2dd30b4f0d0d4d
Lung_Opacity-4052.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4052.png,Lung_Opacity/masks/Lung_Opacity-4052.png,True,G06795,train,a0386114fbba985f9cfb783b5f099585,c03c175b5d5c4a6d
Lung_Opacity-4053.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4053.png,Lung_Opacity/masks/Lung_Opacity-4053.png,True,G06796,train,70d420b8cfe5ee4a616c39f796d4f8b9,c02f3d480bec37d5
Lung_Opacity-4054.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4054.png,Lung_Opacity/masks/Lung_Opacity-4054.png,True,G06797,train,53f0aa4ce42254fd9735cd53107af1e7,d43c6d510b2e3e3a
Lung_Opacity-4055.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4055.png,Lung_Opacity/masks/Lung_Opacity-4055.png,True,G06798,train,deaaa42b22ca478ca08d23f63e0cfee5,817b574a5a594a5b
Lung_Opacity-4056.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4056.png,Lung_Opacity/masks/Lung_Opacity-4056.png,True,G06799,train,28451667505f495cb7d801a1c093e222,8128175a5f79785b
Lung_Opacity-4057.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4057.png,Lung_Opacity/masks/Lung_Opacity-4057.png,True,G06800,train,f627dd1c32e9a01e48895290c6116b66,9c0c4f6b6ceca585
Lung_Opacity-4058.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4058.png,Lung_Opacity/masks/Lung_Opacity-4058.png,True,G06801,train,b843ab1ddc8348684464156fbaa3eeb5,9d6a474e4a417d4d
Lung_Opacity-4059.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4059.png,Lung_Opacity/masks/Lung_Opacity-4059.png,True,G06802,train,7b1d381631ec6852842b8e0e9697abd6,917a061e1a196dbf
Lung_Opacity-406.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-406.png,Lung_Opacity/masks/Lung_Opacity-406.png,True,G06803,val,b1095014ace30f66f9c40f4f8e6e54b5,d43dc1990f0f0e2f
Lung_Opacity-4060.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4060.png,Lung_Opacity/masks/Lung_Opacity-4060.png,True,G06804,test,e7db813fbee1c598a459b74d4e5f6e94,c97d96535a0a5b1a
Lung_Opacity-4061.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4061.png,Lung_Opacity/masks/Lung_Opacity-4061.png,True,G06805,train,f910b570dfe228f93f6ace00448794d6,d85917c71a1a1dab
Lung_Opacity-4062.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4062.png,Lung_Opacity/masks/Lung_Opacity-4062.png,True,G06806,train,adde31e54624fc909c578a40f63943a3,887f45c64e4e49e5
Lung_Opacity-4063.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4063.png,Lung_Opacity/masks/Lung_Opacity-4063.png,True,G06807,val,8fe287c5025dee6f6cdf3e19847dc1c4,97204e4d1e1f4e9e
Lung_Opacity-4064.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4064.png,Lung_Opacity/masks/Lung_Opacity-4064.png,True,G06808,train,66af0fdcf3b201422d2f40e6aca1c665,c97b55861e18595d
Lung_Opacity-4065.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4065.png,Lung_Opacity/masks/Lung_Opacity-4065.png,True,G06809,train,cbc769c9456cdeb8652482e159b01d88,c91916161b5d5f1e
Lung_Opacity-4066.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4066.png,Lung_Opacity/masks/Lung_Opacity-4066.png,True,G06810,test,8a933155ade7decf8bd7407fd9441457,9919c0961e7b786b
Lung_Opacity-4067.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4067.png,Lung_Opacity/masks/Lung_Opacity-4067.png,True,G06811,train,5bd4f5dc70ba23aa6668ee2f464f9fee,d46b055a1c4a3f7a
Lung_Opacity-4068.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4068.png,Lung_Opacity/masks/Lung_Opacity-4068.png,True,G06812,train,f71c52292ea56687f9477b01a766dea1,946e474b0b6f2c6c
Lung_Opacity-4069.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4069.png,Lung_Opacity/masks/Lung_Opacity-4069.png,True,G06813,train,a201374543ff8d3e6890cb187d30816f,917c06d7930b4b1f
Lung_Opacity-407.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-407.png,Lung_Opacity/masks/Lung_Opacity-407.png,True,G06814,train,40e74fe7ac011fe38fdcfe6803c15b5b,c83f532ca4e3d5c8
Lung_Opacity-4070.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4070.png,Lung_Opacity/masks/Lung_Opacity-4070.png,True,G06815,train,bbb33d5af4e95fcbff4c9876dbe370e8,cc35d34b0e2d6c3c
Lung_Opacity-4071.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4071.png,Lung_Opacity/masks/Lung_Opacity-4071.png,True,G06816,train,b8321bad8e66abf3dc83a62d8ca7a1a6,847c2b71e08e9fd8
Lung_Opacity-4072.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4072.png,Lung_Opacity/masks/Lung_Opacity-4072.png,True,G06817,train,74a7702d9b94c8c3664a86161b11e8ac,d12a161b791b7d86
Lung_Opacity-4073.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4073.png,Lung_Opacity/masks/Lung_Opacity-4073.png,True,G06818,train,530c1005b72e552eebbcc7c5bcc1f30b,897f041e1a5f5a5a
Lung_Opacity-4074.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4074.png,Lung_Opacity/masks/Lung_Opacity-4074.png,True,G06819,train,03ccd7ff4bc73ef4376f5a1a1f7f89a8,81785f867a99689d
Lung_Opacity-4075.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4075.png,Lung_Opacity/masks/Lung_Opacity-4075.png,True,G06820,train,150296b9c44de9604326417cbe784032,9579c11e5ec80b1f
Lung_Opacity-4076.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4076.png,Lung_Opacity/masks/Lung_Opacity-4076.png,True,G06821,val,ecb5288b43aa183fb33c549065e1e00f,dc3ec25b4a0f0b4b
Lung_Opacity-4077.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4077.png,Lung_Opacity/masks/Lung_Opacity-4077.png,True,G06822,val,0360b4f2b623a12f82ffe779203a2952,d029530a2e3e3f37
Lung_Opacity-4078.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4078.png,Lung_Opacity/masks/Lung_Opacity-4078.png,True,G06823,train,b7e2f7a37f3ab75a8cc2d1edda062c9a,c52e164d4c6c6dec
Lung_Opacity-4079.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4079.png,Lung_Opacity/masks/Lung_Opacity-4079.png,True,G06824,test,1e4e7643d766a8cbd2d7ca2ed580aac8,c14a861a1b5b5b5f
Lung_Opacity-408.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-408.png,Lung_Opacity/masks/Lung_Opacity-408.png,True,G06825,test,293b1bee0469538c3eba39541b7fc4a9,856ea2c14d64767d
Lung_Opacity-4080.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4080.png,Lung_Opacity/masks/Lung_Opacity-4080.png,True,G06826,train,d4ae91084df1ddeddf2d65b965e65b20,c44d0e0e6c6e6c7e
Lung_Opacity-4081.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4081.png,Lung_Opacity/masks/Lung_Opacity-4081.png,True,G06827,train,e344cf1b4103c8edec7d7b015c80127d,8939061e195d5f5d
Lung_Opacity-4082.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4082.png,Lung_Opacity/masks/Lung_Opacity-4082.png,True,G06828,train,c628c5d38028c9e02144d81b2097e7e2,856e065f786c6c6c
Lung_Opacity-4083.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4083.png,Lung_Opacity/masks/Lung_Opacity-4083.png,True,G06829,train,52886a08348a9adf8c37805cfccbd566,916e45471a5b4f4b
Lung_Opacity-4084.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4084.png,Lung_Opacity/masks/Lung_Opacity-4084.png,True,G06830,train,7a8f5f6d7fdb5c907e4ba66615525e85,cc060a6a6d6fe46e
Lung_Opacity-4085.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4085.png,Lung_Opacity/masks/Lung_Opacity-4085.png,True,G06831,train,04af5b4a1162cba645b3033be54a22e1,d435030b0d6f2f6d
Lung_Opacity-4086.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4086.png,Lung_Opacity/masks/Lung_Opacity-4086.png,True,G06832,train,4cf663d316583a4e1d4a440e5f5f216f,9d3cc21b8a0eb8af
Lung_Opacity-4087.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4087.png,Lung_Opacity/masks/Lung_Opacity-4087.png,True,G06833,train,505314ccadcb796bde322b0c862792a2,9d2ac00f921b2ddf
Lung_Opacity-4088.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4088.png,Lung_Opacity/masks/Lung_Opacity-4088.png,True,G06834,val,2d1c55c6b48401a1d4103b8242651956,c03b56105b1b7b3b
Lung_Opacity-4089.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4089.png,Lung_Opacity/masks/Lung_Opacity-4089.png,True,G06835,val,6b7720a93d0de0b837e2a37b29a9f20e,9568168e1e1b1f5b
Lung_Opacity-409.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-409.png,Lung_Opacity/masks/Lung_Opacity-409.png,True,G06836,train,f02769ffed466c5bf2a1e41682c0a490,852f075d496ad2e3
Lung_Opacity-4090.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4090.png,Lung_Opacity/masks/Lung_Opacity-4090.png,True,G06837,test,addae28c8cae0ebce319f6e1db414029,c43f7b4d0b4a2cca
Lung_Opacity-4091.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4091.png,Lung_Opacity/masks/Lung_Opacity-4091.png,True,G06838,train,c55468d9c0f5879678e9e038972d3cf6,d03d5a173a1c3cb6
Lung_Opacity-4092.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4092.png,Lung_Opacity/masks/Lung_Opacity-4092.png,True,G06839,train,0efe0ff24f781e46665085b79ecf1808,c13d4a1c0db0afb7
Lung_Opacity-4093.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4093.png,Lung_Opacity/masks/Lung_Opacity-4093.png,True,G06840,train,e57e3718a4d9f67d4505a02963286df7,83710d9c333b5b59
Lung_Opacity-4094.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4094.png,Lung_Opacity/masks/Lung_Opacity-4094.png,True,G06841,train,baa9dcb210b22eb60dbbc0f4430955c4,d92906161c5f137f
Lung_Opacity-4095.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4095.png,Lung_Opacity/masks/Lung_Opacity-4095.png,True,G06842,test,3ba6356bd651c9afa919dc4c95e70663,91781a181f7b5b1b
Lung_Opacity-4096.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4096.png,Lung_Opacity/masks/Lung_Opacity-4096.png,True,G06843,val,d8198dd80a039f58bfba91a32d85975c,981a930a3caaafaf
Lung_Opacity-4097.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4097.png,Lung_Opacity/masks/Lung_Opacity-4097.png,True,G06844,test,2d8bc2cf3368992e67fe155ac6a8c882,d94dc61e49492f2b
Lung_Opacity-4098.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4098.png,Lung_Opacity/masks/Lung_Opacity-4098.png,True,G06845,train,62a731cad1df15f07c37f0fc9b01dc25,807d1f969a0b0d5f
Lung_Opacity-4099.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4099.png,Lung_Opacity/masks/Lung_Opacity-4099.png,True,G06846,train,f55cb3d04e4c7ec088fb83fdcb310084,837a541f131b7a78
Lung_Opacity-41.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-41.png,Lung_Opacity/masks/Lung_Opacity-41.png,True,G06847,train,9d7b047183dd98ae763feadd24389c7b,807f87061a1d5d5f
Lung_Opacity-410.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-410.png,Lung_Opacity/masks/Lung_Opacity-410.png,True,G06848,train,37f3bf268c72ba9de8755ea6127fedde,997bc6c61a564b19
Lung_Opacity-4100.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4100.png,Lung_Opacity/masks/Lung_Opacity-4100.png,True,G06849,train,cb110e4f20de9d2fa39c841cab1d56b9,9f60ae591b6b68c8
Lung_Opacity-4101.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4101.png,Lung_Opacity/masks/Lung_Opacity-4101.png,True,G06850,train,d65bcab1dcdb13ddb20cf7370a19a95b,8c6e033a692b3ba7
Lung_Opacity-4102.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4102.png,Lung_Opacity/masks/Lung_Opacity-4102.png,True,G06851,train,fd40eb8cb87345f5e628d2b8cf823347,c83f834b1d6c1b4d
Lung_Opacity-4103.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4103.png,Lung_Opacity/masks/Lung_Opacity-4103.png,True,G06852,train,b52f824d4c919b6a4080c802e14b7f4b,d67827986a976c94
Lung_Opacity-4104.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4104.png,Lung_Opacity/masks/Lung_Opacity-4104.png,True,G06853,train,f232a08fe2146589c72ac969a154b9d9,dd2a431b5d1c386d
Lung_Opacity-4105.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4105.png,Lung_Opacity/masks/Lung_Opacity-4105.png,True,G06854,train,ebe8cbaf1dfe2c085e4500f4a3ea4f3b,cd3d12434f1c596d
Lung_Opacity-4106.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4106.png,Lung_Opacity/masks/Lung_Opacity-4106.png,True,G06855,train,7d48ca793478902a3fcc12789df169d9,9979561e19283bb5
Lung_Opacity-4107.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4107.png,Lung_Opacity/masks/Lung_Opacity-4107.png,True,G06856,train,98eb613c8f10df1b9a8acb6274257418,c42e515f0d4e4e6e
Lung_Opacity-4108.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4108.png,Lung_Opacity/masks/Lung_Opacity-4108.png,True,G06857,train,c92ad7e4feb84a4e021223d1d112eea6,997c861a0f1d396d
Lung_Opacity-4109.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4109.png,Lung_Opacity/masks/Lung_Opacity-4109.png,True,G06858,train,83c479cbe45eddb1b80333dd07897c67,9169475e1a5a5b33
Lung_Opacity-411.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-411.png,Lung_Opacity/masks/Lung_Opacity-411.png,True,G06859,val,0b9d2431320b6380c786113e9d3adbd1,cc3d0a0db493d3da
Lung_Opacity-4110.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4110.png,Lung_Opacity/masks/Lung_Opacity-4110.png,True,G06860,val,3c54e51d43dc8d57cc47dfa7441fdb10,92337630b5b5a58d
Lung_Opacity-4111.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4111.png,Lung_Opacity/masks/Lung_Opacity-4111.png,True,G06861,val,b3433d16685e1b7dfefea43a95160d51,8378041c1b7f1f5e
Lung_Opacity-4112.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4112.png,Lung_Opacity/masks/Lung_Opacity-4112.png,True,G06862,test,0b21850ef65882a62d694b589d3617ba,c2642f6d3c383b39
Lung_Opacity-4113.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4113.png,Lung_Opacity/masks/Lung_Opacity-4113.png,True,G06863,test,df89ba726f19ca750ee3bdab8a13a48d,c93a961e5c0a693f
Lung_Opacity-4114.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4114.png,Lung_Opacity/masks/Lung_Opacity-4114.png,True,G06864,test,14f8338df516bdd67878d71442b87670,9c2ec61c4d4e6ec6
Lung_Opacity-4115.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4115.png,Lung_Opacity/masks/Lung_Opacity-4115.png,True,G06865,train,70e267df4b7d68acb171cbcd07ed36dd,856e064b6a67e3e1
Lung_Opacity-4116.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4116.png,Lung_Opacity/masks/Lung_Opacity-4116.png,True,G06866,train,976791bade2c1fb42933643aaf7e8535,dd20061f061e7d5f
Lung_Opacity-4117.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4117.png,Lung_Opacity/masks/Lung_Opacity-4117.png,True,G06867,train,e83dfd21b78d924e4d16f0c9f3eb6b2d,993b861e18595b5d
Lung_Opacity-4118.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4118.png,Lung_Opacity/masks/Lung_Opacity-4118.png,True,G06868,train,df62dfebebe666e0fdf43ce64ed402c0,c4bd4bcb2fc82d84
Lung_Opacity-4119.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4119.png,Lung_Opacity/masks/Lung_Opacity-4119.png,True,G06869,val,07d976f838ac4184bc8372db29f0d362,9969861e174e7a39
Lung_Opacity-412.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-412.png,Lung_Opacity/masks/Lung_Opacity-412.png,True,G06870,train,ae4707f54b7247291820593046b84910,843fd313125b3d1e
Lung_Opacity-4120.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4120.png,Lung_Opacity/masks/Lung_Opacity-4120.png,True,G06871,train,ca22605161f32ecbc67698726e450691,995b44175e4b1d4e
Lung_Opacity-4121.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4121.png,Lung_Opacity/masks/Lung_Opacity-4121.png,True,G06872,train,7ee94120fd781e01f12c5193dc23b1f9,997b431f1a491f1c
Lung_Opacity-4122.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4122.png,Lung_Opacity/masks/Lung_Opacity-4122.png,True,G06873,train,2a0ffb88fa7e1879f03fa537947f51c8,997dc61a4d4f1949
Lung_Opacity-4123.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4123.png,Lung_Opacity/masks/Lung_Opacity-4123.png,True,G06874,train,aadcec5bc19b5f98e033444d2a379da0,c92de44b91e5469e
Lung_Opacity-4124.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4124.png,Lung_Opacity/masks/Lung_Opacity-4124.png,True,G06875,train,18f72b4685b940068fc498cb9229f840,817f86173d3c0d1d
Lung_Opacity-4125.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4125.png,Lung_Opacity/masks/Lung_Opacity-4125.png,True,G06876,val,7d2a08ce6bf0f878d8f46ab54586bcd5,9d788696b81fb838
Lung_Opacity-4126.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4126.png,Lung_Opacity/masks/Lung_Opacity-4126.png,True,G06877,train,feece2d2ade919251cc0ebe91d87cc82,8178161e5f1c5e79
Lung_Opacity-4127.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4127.png,Lung_Opacity/masks/Lung_Opacity-4127.png,True,G06878,test,f4a746967ecb92e1e863792e8b8bcea3,91681e5e195a5b5b
Lung_Opacity-4128.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4128.png,Lung_Opacity/masks/Lung_Opacity-4128.png,True,G06879,train,3b7dd66439c8372a3595b809a1e18ae2,cc7d4b0b0f2d1c1d
Lung_Opacity-4129.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4129.png,Lung_Opacity/masks/Lung_Opacity-4129.png,True,G06880,train,16ef33aaae06752df5ffac8555cb9224,dc2913574c2cfde0
Lung_Opacity-413.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-413.png,Lung_Opacity/masks/Lung_Opacity-413.png,True,G06881,train,6ae99c1197ae897a8b2bd2a960a6443a,913d465e5d493c53
Lung_Opacity-4130.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4130.png,Lung_Opacity/masks/Lung_Opacity-4130.png,True,G06882,val,3af165fa61410483a0a4e9596635186e,d52a470f0d0d4f4f
Lung_Opacity-4131.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4131.png,Lung_Opacity/masks/Lung_Opacity-4131.png,True,G06883,val,ac6151342f3a1a4459972dd9d2eb98cf,817b841e187b7b39
Lung_Opacity-4132.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4132.png,Lung_Opacity/masks/Lung_Opacity-4132.png,True,G06884,train,8979c0088f124554758357340b289d69,cd6ec34f4d444d07
Lung_Opacity-4133.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4133.png,Lung_Opacity/masks/Lung_Opacity-4133.png,True,G06885,test,b8f318fb4de1f1f88d87c3c36f77e98d,d17f861e5a4a1a78
Lung_Opacity-4134.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4134.png,Lung_Opacity/masks/Lung_Opacity-4134.png,True,G06886,train,92110a0fa4971d9f69be124c4c723386,991a961a16173cef
Lung_Opacity-4135.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4135.png,Lung_Opacity/masks/Lung_Opacity-4135.png,True,G06887,test,2c0f0342a0e0bd6da466d3a2808d0906,c82d875a454f396d
Lung_Opacity-4136.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4136.png,Lung_Opacity/masks/Lung_Opacity-4136.png,True,G06888,test,2dc3f8db8bb889e531589cb76a6b126c,857e86801f5d3e2d
Lung_Opacity-4137.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4137.png,Lung_Opacity/masks/Lung_Opacity-4137.png,True,G06889,test,9f5361c227c2288a5e3ace3e422e9108,803d485d1b2f5ea7
Lung_Opacity-4138.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4138.png,Lung_Opacity/masks/Lung_Opacity-4138.png,True,G06890,train,6baf0b19c18d837f15ea6a116426c78d,c03c070f1e4e4f5f
Lung_Opacity-4139.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4139.png,Lung_Opacity/masks/Lung_Opacity-4139.png,True,G06891,train,62fe4b5e36e8c0e6c7529f8ab585deb4,d83a921a1e18fef2
Lung_Opacity-414.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-414.png,Lung_Opacity/masks/Lung_Opacity-414.png,True,G06892,train,a2ca9cea9db6348cd68e30b09bdae0dc,9d6ec2524b471a5b
Lung_Opacity-4140.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4140.png,Lung_Opacity/masks/Lung_Opacity-4140.png,True,G06893,test,078d1b4c8682fdbd2cc392dd8230952d,896f864249edf478
Lung_Opacity-4141.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4141.png,Lung_Opacity/masks/Lung_Opacity-4141.png,True,G06894,train,0ddc7861ce99652734d6293dd2f6a169,d93b460a5d4b1b0f
Lung_Opacity-4142.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4142.png,Lung_Opacity/masks/Lung_Opacity-4142.png,True,G06895,train,d07c50f09fe5fab7bbd0a61a9ec82e04,dc2c570b5f4c0c2f
Lung_Opacity-4143.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4143.png,Lung_Opacity/masks/Lung_Opacity-4143.png,True,G06896,train,a1ee5ed6deb29f36a2cbb9cfb343dadf,9168ce145f1b5b4b
Lung_Opacity-4144.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4144.png,Lung_Opacity/masks/Lung_Opacity-4144.png,True,G06897,train,911ead63a75684be87f3aea8169992df,d46e0b1a4c6c7cf8
Lung_Opacity-4145.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4145.png,Lung_Opacity/masks/Lung_Opacity-4145.png,True,G06898,train,f5916d81fa854c1b5e631b3773ccf2c4,957c07574a5e6c68
Lung_Opacity-4146.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4146.png,Lung_Opacity/masks/Lung_Opacity-4146.png,True,G06899,train,7173c15bd225f20bca8f29104fc95b71,894d8e7b61c682af
Lung_Opacity-4147.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4147.png,Lung_Opacity/masks/Lung_Opacity-4147.png,True,G06900,train,b2281c1d68fd7f0dcc4f02bb792a749e,dc3cc3535f4c5858
Lung_Opacity-4148.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4148.png,Lung_Opacity/masks/Lung_Opacity-4148.png,True,G06901,train,62ea166f68af9dc482affb9c892080df,817e82175f0d7c6c
Lung_Opacity-4149.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4149.png,Lung_Opacity/masks/Lung_Opacity-4149.png,True,G06902,train,8380026f61e97a12fd641afea85f5b2f,dc28d75b585a5a4a
Lung_Opacity-415.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-415.png,Lung_Opacity/masks/Lung_Opacity-415.png,True,G06903,val,46e3199e877d27cc2eef0399e54bb042,c52a530b2d3c2dbb
Lung_Opacity-4150.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4150.png,Lung_Opacity/masks/Lung_Opacity-4150.png,True,G06904,val,4b77e754d2a59d0067a2088fdf8b74f0,dc2e431b095f1c6d
Lung_Opacity-4151.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4151.png,Lung_Opacity/masks/Lung_Opacity-4151.png,True,G06905,val,b77355012860093505336c92dfccbd8b,917b46161a4f9f19
Lung_Opacity-4152.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4152.png,Lung_Opacity/masks/Lung_Opacity-4152.png,True,G06906,test,56cc88f7ba9f4705d0a260fe060127a2,dd68d15b494d5c58
Lung_Opacity-4153.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4153.png,Lung_Opacity/masks/Lung_Opacity-4153.png,True,G06907,train,91317438a04c5b6156d6cedf3c6d003b,817856865f5b3a39
Lung_Opacity-4154.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4154.png,Lung_Opacity/masks/Lung_Opacity-4154.png,True,G06908,train,b32c169611e8a80abfe28f5023509ac5,c929964e5e187a5b
Lung_Opacity-4155.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4155.png,Lung_Opacity/masks/Lung_Opacity-4155.png,True,G06909,train,32bac00a40f6519e93ef2bca61e861bf,9c7e411f0e0c3eae
Lung_Opacity-4156.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4156.png,Lung_Opacity/masks/Lung_Opacity-4156.png,True,G06910,train,85191027c67c1252e654f80621ff2412,817d065f197b5878
Lung_Opacity-4157.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4157.png,Lung_Opacity/masks/Lung_Opacity-4157.png,True,G06911,train,8031c09df1c3bade48675bb78a5628e4,d03b42515c3b1f1f
Lung_Opacity-4158.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4158.png,Lung_Opacity/masks/Lung_Opacity-4158.png,True,G06912,train,a7f68ce12cf1e97df03b26e5e93c7b74,d03fc202197d7979
Lung_Opacity-4159.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4159.png,Lung_Opacity/masks/Lung_Opacity-4159.png,True,G06913,train,b084cb2cd8796dbc667d903e4385909a,dd2cc20b97471f1c
Lung_Opacity-416.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-416.png,Lung_Opacity/masks/Lung_Opacity-416.png,True,G06914,train,591b60d6d41dcf9a293d79614d9713ac,d43e4acb4a6a6a6a
Lung_Opacity-4160.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4160.png,Lung_Opacity/masks/Lung_Opacity-4160.png,True,G06915,val,4f761a7f89499e68e9326fb870bf0be1,d845961a5f1a5a5b
Lung_Opacity-4161.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4161.png,Lung_Opacity/masks/Lung_Opacity-4161.png,True,G06916,train,4681d1a2fc91788a002eec0ef8a9d131,d42f420f0d3f3a33
Lung_Opacity-4162.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4162.png,Lung_Opacity/masks/Lung_Opacity-4162.png,True,G06917,train,a30aa85927b83afb8d0c1aa91e6dea13,c56603490f6f6e6a
Lung_Opacity-4163.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4163.png,Lung_Opacity/masks/Lung_Opacity-4163.png,True,G06918,train,f422a120ab2d00ec5fc6322dda695bc5,83788d141f3b1b79
Lung_Opacity-4164.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4164.png,Lung_Opacity/masks/Lung_Opacity-4164.png,True,G06919,test,063fef49561f7cc796884380a8fd88d0,813b94941b5f1f1d
Lung_Opacity-4165.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4165.png,Lung_Opacity/masks/Lung_Opacity-4165.png,True,G06920,train,5b191152d052e7f2c9a51e567babb148,856e0f586ee0c6d3
Lung_Opacity-4166.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4166.png,Lung_Opacity/masks/Lung_Opacity-4166.png,True,G06921,train,c35ab1b604f1c5d9092b60ce1eed7f61,d42e074f2e2e0f0f
Lung_Opacity-4167.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4167.png,Lung_Opacity/masks/Lung_Opacity-4167.png,True,G06922,train,3592f74769d029c93672882b3dd0bad0,de155b5a0a2a1e9d
Lung_Opacity-4168.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4168.png,Lung_Opacity/masks/Lung_Opacity-4168.png,True,G06923,train,3b3a789d3a7ade414f29ab3e9cf64003,c46f03190e4c6eef
Lung_Opacity-4169.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4169.png,Lung_Opacity/masks/Lung_Opacity-4169.png,True,G06924,val,7f5867bf5285216dfb6388ef3309d2fd,9978051e5a595f78
Lung_Opacity-417.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-417.png,Lung_Opacity/masks/Lung_Opacity-417.png,True,G06925,val,41c5071e2df83ca3231354ca85a4841c,81281e5f787a6b63
Lung_Opacity-4170.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4170.png,Lung_Opacity/masks/Lung_Opacity-4170.png,True,G06926,test,b1cf8be48063a6e0cc2eba7d77de00f8,907f034f0f1c3c4f
Lung_Opacity-4171.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4171.png,Lung_Opacity/masks/Lung_Opacity-4171.png,True,G06927,train,db029936b4e03fc4bccec894340985d0,d02d5d0a5a2d7d2d
Lung_Opacity-4172.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4172.png,Lung_Opacity/masks/Lung_Opacity-4172.png,True,G06928,train,e625a9917495574dbae3bf1f385e8f9f,d42ec7cf096d2c2c
Lung_Opacity-4173.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4173.png,Lung_Opacity/masks/Lung_Opacity-4173.png,True,G06929,train,9624e593d651034963720268c8c1e58f,917bc6961c105f1f
Lung_Opacity-4174.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4174.png,Lung_Opacity/masks/Lung_Opacity-4174.png,True,G06930,train,ec12810992cbd106bcd37e58ddeab304,943e879d1d4d1c1e
Lung_Opacity-4175.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4175.png,Lung_Opacity/masks/Lung_Opacity-4175.png,True,G06931,train,5a0003f7937d0e39b0797c5ad34c1897,c43fdbc10ec81f68
Lung_Opacity-4176.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4176.png,Lung_Opacity/masks/Lung_Opacity-4176.png,True,G06932,train,54de35cac34b3c17c777b21bfbcdbbbd,dc2f434c2c2cfcf0
Lung_Opacity-4177.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4177.png,Lung_Opacity/masks/Lung_Opacity-4177.png,True,G06933,train,24fcbcb01bbe33ca2eba0bedf8ef7051,9979851e592d3aa9
Lung_Opacity-4178.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4178.png,Lung_Opacity/masks/Lung_Opacity-4178.png,True,G06934,test,5d0e42271f632582f323eadd28a0fb8b,8d4e067d4d7a4aca
Lung_Opacity-4179.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4179.png,Lung_Opacity/masks/Lung_Opacity-4179.png,True,G06935,train,3e00ad090b2a2f78a54a1333ae4aa5b1,d96c871a1f461b4d
Lung_Opacity-418.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-418.png,Lung_Opacity/masks/Lung_Opacity-418.png,True,G06936,train,09a2cfdb2e76dd26f2d911391fcdaa5b,d40a574b1a2e2f2f
Lung_Opacity-4180.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4180.png,Lung_Opacity/masks/Lung_Opacity-4180.png,True,G06937,train,fab57c4b4cfd89c2be5441204a537326,95681e1e5a5c7879
Lung_Opacity-4181.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4181.png,Lung_Opacity/masks/Lung_Opacity-4181.png,True,G06938,train,4c6090a56751818fbfac7e013ee994f1,d47d434b0c1e4f2d
Lung_Opacity-4182.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4182.png,Lung_Opacity/masks/Lung_Opacity-4182.png,True,G06939,train,c77dd1db9e167873501ca93de91dfba0,c52a1a185df8dae3
Lung_Opacity-4183.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4183.png,Lung_Opacity/masks/Lung_Opacity-4183.png,True,G06940,train,adb87bc5606d20b90e93a9a7a5076f1a,c42d424f0e7f0e3d
Lung_Opacity-4184.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4184.png,Lung_Opacity/masks/Lung_Opacity-4184.png,True,G06941,train,1e834b5ea9352dbd4eec3f06167eea7f,a94f861e4c5d7a68
Lung_Opacity-4185.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4185.png,Lung_Opacity/masks/Lung_Opacity-4185.png,True,G06942,val,484b43cd3189feea839b4b2617dec394,884e8698faf3a3f0
Lung_Opacity-4186.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4186.png,Lung_Opacity/masks/Lung_Opacity-4186.png,True,G06943,val,b5a2e7176b364b8d40a21c8c1d864c67,8a39020f4f6f783b
Lung_Opacity-4187.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4187.png,Lung_Opacity/masks/Lung_Opacity-4187.png,True,G06944,val,1dc1cd7ea59aa646bea1dfe7d020aa7a,c169e7c6d61ad503
Lung_Opacity-4188.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4188.png,Lung_Opacity/masks/Lung_Opacity-4188.png,True,G06945,test,b35d045bb031bacdd2402847537d14cc,c13f9212154f5d5b
Lung_Opacity-4189.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4189.png,Lung_Opacity/masks/Lung_Opacity-4189.png,True,G06946,test,751d21ba32d276dddbf89929708b7f53,917ac71e58495d5a
Lung_Opacity-419.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-419.png,Lung_Opacity/masks/Lung_Opacity-419.png,True,G06947,train,c7c2851c3024767aa8c01c0c4129371f,d93bd61a4d5e5908
Lung_Opacity-4190.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4190.png,Lung_Opacity/masks/Lung_Opacity-4190.png,True,G06948,train,a73c350142a18d697e5a2fa410b79671,8a3d8302bdb4f593
Lung_Opacity-4191.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4191.png,Lung_Opacity/masks/Lung_Opacity-4191.png,True,G06949,train,1a7b2abcc719bdf6a8af08fbede9cc12,d22dc3431f0f0f0f
Lung_Opacity-4192.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4192.png,Lung_Opacity/masks/Lung_Opacity-4192.png,True,G06950,train,21f0e352bb395d8a09127cdf85464940,c02db492cb497d7c
Lung_Opacity-4193.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4193.png,Lung_Opacity/masks/Lung_Opacity-4193.png,True,G06951,train,b7a35173ab1bb4c990e20ef1483af36c,8b29929f1a3958ed
Lung_Opacity-4194.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4194.png,Lung_Opacity/masks/Lung_Opacity-4194.png,True,G06952,train,6cbca75dd013f8386a1963d3786bd5d2,a82f861a094d5fcf
Lung_Opacity-4195.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4195.png,Lung_Opacity/masks/Lung_Opacity-4195.png,True,G06953,train,5759dc1dbe683f954a714a1d4abc54d5,c92856d30b8d2fba
Lung_Opacity-4196.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4196.png,Lung_Opacity/masks/Lung_Opacity-4196.png,True,G06954,train,70423c647f10ac913c9a1d5cc16c4a5c,c42f0742ea63e4f9
Lung_Opacity-4197.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4197.png,Lung_Opacity/masks/Lung_Opacity-4197.png,True,G06955,train,77ef4af6fdc4e59e35ddcfe3e04a3f7e,c87dd7425c485a5d
Lung_Opacity-4198.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4198.png,Lung_Opacity/masks/Lung_Opacity-4198.png,True,G06956,train,3b52e1b958c39b87fc74dbd3c7934ed0,dc3f431e4d491e0b
Lung_Opacity-4199.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4199.png,Lung_Opacity/masks/Lung_Opacity-4199.png,True,G06957,train,acd88d044aadaf5f2a3f8ff8021cd54a,d82dd7c35d0d183a
Lung_Opacity-42.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-42.png,Lung_Opacity/masks/Lung_Opacity-42.png,True,G06958,val,035ab9175a8fb5a7e53f9ccda6854e86,cc0fc31d481ac9fb
Lung_Opacity-420.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-420.png,Lung_Opacity/masks/Lung_Opacity-420.png,True,G06959,test,6e47216733485a1d99b28e895087602d,917b061e1a1f5b1b
Lung_Opacity-4200.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4200.png,Lung_Opacity/masks/Lung_Opacity-4200.png,True,G06960,train,33a2be2b9751104baf45866065b27cf3,9128465e1b5b795b
Lung_Opacity-4201.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4201.png,Lung_Opacity/masks/Lung_Opacity-4201.png,True,G06961,train,a9800a270236fb7b26e7ff81de9291ee,c81f025d3c6b3ce6
Lung_Opacity-4202.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4202.png,Lung_Opacity/masks/Lung_Opacity-4202.png,True,G06962,test,598796811ff2942601a28f2db4a67d73,9979c71e184d39b8
Lung_Opacity-4203.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4203.png,Lung_Opacity/masks/Lung_Opacity-4203.png,True,G06963,train,82ff9c3c992dc6c176cd275034f85f82,913a101dbbbbaca5
Lung_Opacity-4204.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4204.png,Lung_Opacity/masks/Lung_Opacity-4204.png,True,G06964,val,dc4ae060a063385412902e0ca9c8a35b,813e061a1b1f7e5e
Lung_Opacity-4205.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4205.png,Lung_Opacity/masks/Lung_Opacity-4205.png,True,G06965,train,a58cce30f7d2ee558c70a593b59fb86c,dd2a474c4c4cd2f3
Lung_Opacity-4206.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4206.png,Lung_Opacity/masks/Lung_Opacity-4206.png,True,G06966,train,9aee63acee9def194b4caa7775769886,dd2ca2539742bb51
Lung_Opacity-4207.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4207.png,Lung_Opacity/masks/Lung_Opacity-4207.png,True,G06967,val,104729964bcccd0f4f402853786a14eb,9b78161a1e1e1a3f
Lung_Opacity-4208.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4208.png,Lung_Opacity/masks/Lung_Opacity-4208.png,True,G06968,test,4b9e5ce7a2d7b32b01a7905889ac3ad3,dc4a431d3838bfa3
Lung_Opacity-4209.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4209.png,Lung_Opacity/masks/Lung_Opacity-4209.png,True,G06969,train,907a27fcff60907a85a6893623af3819,d43cc3031b0b1f5f
Lung_Opacity-421.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-421.png,Lung_Opacity/masks/Lung_Opacity-421.png,True,G06970,train,4f2103773cb7a65d694fb3052355a2c9,d03d4b923ec33ec2
Lung_Opacity-4210.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4210.png,Lung_Opacity/masks/Lung_Opacity-4210.png,True,G06971,train,05bf4bec088c32e20664f1e4737b2ca5,d52a07171f0d0f1f
Lung_Opacity-4211.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4211.png,Lung_Opacity/masks/Lung_Opacity-4211.png,True,G06972,train,7293f564e67d57d9c679aa07ddd43dc2,807f0f180d5d4f4f
Lung_Opacity-4212.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4212.png,Lung_Opacity/masks/Lung_Opacity-4212.png,True,G06973,train,2556837d3266d427483f3222110d176d,d03f550a0f0b2f2f
Lung_Opacity-4213.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4213.png,Lung_Opacity/masks/Lung_Opacity-4213.png,True,G06974,train,aa3a1628eb33ba60aa21e98b5561c5bc,c53cb3c39506dc1d
Lung_Opacity-4214.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4214.png,Lung_Opacity/masks/Lung_Opacity-4214.png,True,G06975,train,2f228ef70fc5adae0a09fdd5f5bd3f4c,d93f46481c6c3b65
Lung_Opacity-4215.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4215.png,Lung_Opacity/masks/Lung_Opacity-4215.png,True,G06976,train,41ae12ebd5fd01fcc1b2ec14d5e47b98,897b8616594b5b59
Lung_Opacity-4216.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4216.png,Lung_Opacity/masks/Lung_Opacity-4216.png,True,G06977,train,74c9e6bacc462e15f7c333f1ef63d109,d13b1786df087d09
Lung_Opacity-4217.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4217.png,Lung_Opacity/masks/Lung_Opacity-4217.png,True,G06978,train,67829f6e22fcd06ec9179e4c1eda7343,d43ee51d944e2c56
Lung_Opacity-4218.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4218.png,Lung_Opacity/masks/Lung_Opacity-4218.png,True,G06979,train,3c5163496b4edd648b3e69eeb7683aeb,d47c0b8d3cd135d1
Lung_Opacity-4219.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4219.png,Lung_Opacity/masks/Lung_Opacity-4219.png,True,G06980,train,802d9a2d0f394e6e60b65d4b7258e85f,d42f0b1c4e2cbbf0
Lung_Opacity-422.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-422.png,Lung_Opacity/masks/Lung_Opacity-422.png,True,G06981,train,67cae12ba0536068fac1a59c2ef2b0f6,c42d130b0c6e5efd
Lung_Opacity-4220.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4220.png,Lung_Opacity/masks/Lung_Opacity-4220.png,True,G06982,val,012a7b48a08f045e1af94c22bc8373b9,937884101b5f735f
Lung_Opacity-4221.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4221.png,Lung_Opacity/masks/Lung_Opacity-4221.png,True,G06983,val,2f9e7a425216a74d964919a5d9bbfd1d,9568065f1f58595d
Lung_Opacity-4222.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4222.png,Lung_Opacity/masks/Lung_Opacity-4222.png,True,G06984,test,7e20e55bb67ecd012699ba600e2cde19,d13906de1a19ef19
Lung_Opacity-4223.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4223.png,Lung_Opacity/masks/Lung_Opacity-4223.png,True,G06985,val,31c8cdb839dad6de514d0bc99e5c6016,d03d53531e1a5d5a
Lung_Opacity-4224.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4224.png,Lung_Opacity/masks/Lung_Opacity-4224.png,True,G06986,test,4a0168c56ae7498539deae24a40c38ca,dc4487190d4efc5b
Lung_Opacity-4225.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4225.png,Lung_Opacity/masks/Lung_Opacity-4225.png,True,G06987,train,de82ebc91fdbc195fbe7573c4e75d410,917b501d0939bf87
Lung_Opacity-4226.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4226.png,Lung_Opacity/masks/Lung_Opacity-4226.png,True,G06988,val,aeba042db929cdf0184b853a5eacfed2,d42bcfcb491d1c1a
Lung_Opacity-4227.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4227.png,Lung_Opacity/masks/Lung_Opacity-4227.png,True,G06989,test,15336d105d8e4cbf7b3cbe4b012a6367,9d0d421f5b491f6a
Lung_Opacity-4228.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4228.png,Lung_Opacity/masks/Lung_Opacity-4228.png,True,G06990,val,4434637acd3a7e14d474c6df16a090f0,d909c71e1e493979
Lung_Opacity-4229.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4229.png,Lung_Opacity/masks/Lung_Opacity-4229.png,True,G06991,train,d552d0274cdd3aa346a87aabeb484b0e,917bc51e12c35b78
Lung_Opacity-423.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-423.png,Lung_Opacity/masks/Lung_Opacity-423.png,True,G06992,train,b72ff54ad7c8e8bb5b848a427c679226,886a871a5d5f1b4d
Lung_Opacity-4230.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4230.png,Lung_Opacity/masks/Lung_Opacity-4230.png,True,G06993,train,aa131113e79afe82334b4901074a4b7f,99015f5a5a7839e9
Lung_Opacity-4231.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4231.png,Lung_Opacity/masks/Lung_Opacity-4231.png,True,G06994,train,42ea63563f3eeadac7ed94d5b0bee816,807b560e1a3b2b3f
Lung_Opacity-4232.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4232.png,Lung_Opacity/masks/Lung_Opacity-4232.png,True,G06995,train,78a9307b4f7ab2880235915db662bc11,d43b0a063bb427f3
Lung_Opacity-4233.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4233.png,Lung_Opacity/masks/Lung_Opacity-4233.png,True,G06996,train,ae123ecaa7c113b3b3bc79f7714496e6,d83913133e68353f
Lung_Opacity-4234.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4234.png,Lung_Opacity/masks/Lung_Opacity-4234.png,True,G06997,train,5f2eb3db9effd5941cf1f340a6e9637f,d978e0921bc95a9e
Lung_Opacity-4235.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4235.png,Lung_Opacity/masks/Lung_Opacity-4235.png,True,G06998,train,b3a9a8090814f8cf947f782c8b68748a,d53f45470d491d1d
Lung_Opacity-4236.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4236.png,Lung_Opacity/masks/Lung_Opacity-4236.png,True,G06999,train,c317e58fef8e6df773167cb549c86e7c,d42bd2d75a524d4a
Lung_Opacity-4237.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4237.png,Lung_Opacity/masks/Lung_Opacity-4237.png,True,G07000,train,98ea55eeb3b3f8842e0902480007ea57,8979861a1e5e3a5d
Lung_Opacity-4238.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4238.png,Lung_Opacity/masks/Lung_Opacity-4238.png,True,G07001,train,90ffd2ecd63f29aa022e67a6bdba3273,95780d4e0d2d4f4f
Lung_Opacity-4239.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4239.png,Lung_Opacity/masks/Lung_Opacity-4239.png,True,G07002,train,3bf03eef66747845cfcc3df6011ef3f8,817f04565e5e5a3a
Lung_Opacity-424.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-424.png,Lung_Opacity/masks/Lung_Opacity-424.png,True,G07003,test,0b62a8db401ab442de8a15d1334495c3,d52e824f5d0c4e5e
Lung_Opacity-4240.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4240.png,Lung_Opacity/masks/Lung_Opacity-4240.png,True,G07004,val,0a4d2c818be7b64232ab358eb5ddfb1f,857e2a0b0f1e2a3f
Lung_Opacity-4241.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4241.png,Lung_Opacity/masks/Lung_Opacity-4241.png,True,G07005,train,9b6b18debccd073a82e4217c63bd863b,d42cc31f4d2e0e6e
Lung_Opacity-4242.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4242.png,Lung_Opacity/masks/Lung_Opacity-4242.png,True,G07006,test,ca4c8e7ff5c8c475e6046c5285c8f670,85280e1e1a1f5f5f
Lung_Opacity-4243.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4243.png,Lung_Opacity/masks/Lung_Opacity-4243.png,True,G07007,test,350f2b8c8ea1cbbe40a2b4782d1926d9,814d9e58797a2e78
Lung_Opacity-4244.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4244.png,Lung_Opacity/masks/Lung_Opacity-4244.png,True,G07008,train,e68f41d37afbe8ef27826b698ecd0272,d63e21c1df4dc2c2
Lung_Opacity-4245.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4245.png,Lung_Opacity/masks/Lung_Opacity-4245.png,True,G07009,train,307fce69ee712ca505ce1344282f163a,cc6d466ee5a78382
Lung_Opacity-4246.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4246.png,Lung_Opacity/masks/Lung_Opacity-4246.png,True,G07010,train,671897cabb7aba584167cc953284c351,916be65618695b59
Lung_Opacity-4247.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4247.png,Lung_Opacity/masks/Lung_Opacity-4247.png,True,G07011,train,50718fb5db34a85f1a8fa98cbe6b07ab,d43e27479b1e1616
Lung_Opacity-4248.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4248.png,Lung_Opacity/masks/Lung_Opacity-4248.png,True,G07012,train,cd2ab43e025feb74aa8b5eac14164c6a,d42d4d4b0e2cadad
Lung_Opacity-4249.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4249.png,Lung_Opacity/masks/Lung_Opacity-4249.png,True,G07013,train,73853060ae10158e0ab5b5c9d029000e,c22f074a2e66e7e1
Lung_Opacity-425.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-425.png,Lung_Opacity/masks/Lung_Opacity-425.png,True,G07014,val,e4212d7d64721c243dade70d62bf2cf6,c42e074d4e43afe3
Lung_Opacity-4250.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4250.png,Lung_Opacity/masks/Lung_Opacity-4250.png,True,G07015,train,a2ffa61117cc9920ed68f3d2b6ddfbc9,803ec7554a5d6a5d
Lung_Opacity-4251.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4251.png,Lung_Opacity/masks/Lung_Opacity-4251.png,True,G07016,train,d8c902187ebbcf474af86eb0bf4877a8,834c057b6b797878
Lung_Opacity-4252.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4252.png,Lung_Opacity/masks/Lung_Opacity-4252.png,True,G07017,train,20f16880a40e77dd543347e3d0ebba36,dc2dc20f5f4c6c2c
Lung_Opacity-4253.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4253.png,Lung_Opacity/masks/Lung_Opacity-4253.png,True,G07018,train,295a7e93e5b664730bd0447d4e93b62c,8a2fc672c8e1eae4
Lung_Opacity-4254.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4254.png,Lung_Opacity/masks/Lung_Opacity-4254.png,True,G07019,train,9131caf2bd1c22b6b3488dd440283a57,c47dd3090f4b1d0f
Lung_Opacity-4255.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4255.png,Lung_Opacity/masks/Lung_Opacity-4255.png,True,G07020,train,fae9416bc2ff0740f0c14b97ce2787f6,d43e855703471f1e
Lung_Opacity-4256.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4256.png,Lung_Opacity/masks/Lung_Opacity-4256.png,True,G07021,train,a2d79d46af001d03e0e6f316dd3e1ec7,997b07165c05fa59
Lung_Opacity-4257.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4257.png,Lung_Opacity/masks/Lung_Opacity-4257.png,True,G07022,train,44656f8c2acae50d2237b21855900158,9d680e184a6fecec
Lung_Opacity-4258.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4258.png,Lung_Opacity/masks/Lung_Opacity-4258.png,True,G07023,train,d50837d003e126a4ac88073d88a15e74,d92b12175b0f3c3a
Lung_Opacity-4259.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4259.png,Lung_Opacity/masks/Lung_Opacity-4259.png,True,G07024,val,8e6c679854095840e4e08d01b4aad02b,887bd65a5958165e
Lung_Opacity-426.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-426.png,Lung_Opacity/masks/Lung_Opacity-426.png,True,G07025,val,f8b5889f5270247093a49f642a744877,d42c430b4e4e4f6f
Lung_Opacity-4260.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4260.png,Lung_Opacity/masks/Lung_Opacity-4260.png,True,G07026,test,0c4a53816e38b1dc02119e1a71a1f30c,882e821f8a1f1f5f
Lung_Opacity-4261.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4261.png,Lung_Opacity/masks/Lung_Opacity-4261.png,True,G07027,test,90346a5be76a31b86701bb27533d14d1,946e07184feaf8c3
Lung_Opacity-4262.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4262.png,Lung_Opacity/masks/Lung_Opacity-4262.png,True,G07028,test,a2e85b1b51d78718580f9586467fe97a,802ad50a7f556b2f
Lung_Opacity-4263.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4263.png,Lung_Opacity/masks/Lung_Opacity-4263.png,True,G07029,train,bb525ea8d114ec7c60f84107033d5b36,d528979a1d385a5d
Lung_Opacity-4264.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4264.png,Lung_Opacity/masks/Lung_Opacity-4264.png,True,G07030,train,60940f1995c1fa577c68de7c838b882e,907f141e5d1a3b1b
Lung_Opacity-4265.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4265.png,Lung_Opacity/masks/Lung_Opacity-4265.png,True,G07031,train,8dbf221ee4fbd7d2f64d12d98e4e5bf5,dd0a125b6939e1b5
Lung_Opacity-4266.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4266.png,Lung_Opacity/masks/Lung_Opacity-4266.png,True,G07032,train,870c06f8966470e92ce5784d407bce79,bb7894165e785a58
Lung_Opacity-4267.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4267.png,Lung_Opacity/masks/Lung_Opacity-4267.png,True,G07033,train,0bbe61751619f907c4ce75e21e50edd2,997f061e5969b0e2
Lung_Opacity-4268.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4268.png,Lung_Opacity/masks/Lung_Opacity-4268.png,True,G07034,train,c5889c593640565b86bfea5d0469b6d7,c22b4b4b4f0b4b2f
Lung_Opacity-4269.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4269.png,Lung_Opacity/masks/Lung_Opacity-4269.png,True,G07035,train,e48cd840b7e9ea09638d390c8d1c0ac1,d03b13125f59595b
Lung_Opacity-427.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-427.png,Lung_Opacity/masks/Lung_Opacity-427.png,True,G07036,train,4219282c277a0388d52f0ac7a741413b,887dc31b0b4b0f2f
Lung_Opacity-4270.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4270.png,Lung_Opacity/masks/Lung_Opacity-4270.png,True,G07037,val,f8823e6e99844faf0deab9b9ed8fdace,942e831f4d1f1d4e
Lung_Opacity-4271.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4271.png,Lung_Opacity/masks/Lung_Opacity-4271.png,True,G07038,train,fb6ab467d37d1f650edbef53d8351321,c83dd1461f4d1b1d
Lung_Opacity-4272.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4272.png,Lung_Opacity/masks/Lung_Opacity-4272.png,True,G07039,train,b23a3b3190d08a6e630f6116e25c46fb,9929055e1a4d7c7e
Lung_Opacity-4273.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4273.png,Lung_Opacity/masks/Lung_Opacity-4273.png,True,G07040,train,e59e6fa883f6bbc36283e89752fed34e,d07d534b5a6a4a2b
Lung_Opacity-4274.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4274.png,Lung_Opacity/masks/Lung_Opacity-4274.png,True,G07041,train,f66ce3ea081656ac04723d95ace33159,dd2a530b0d2c3f0f
Lung_Opacity-4275.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4275.png,Lung_Opacity/masks/Lung_Opacity-4275.png,True,G07042,val,d3ad9a4cf630dbcb5cbd6982ca55bfbc,d625450e2e6b3d78
Lung_Opacity-4276.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4276.png,Lung_Opacity/masks/Lung_Opacity-4276.png,True,G07043,val,82972d205d1c0286ef8a619695f3d1dd,942c0f786d707373
Lung_Opacity-4277.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4277.png,Lung_Opacity/masks/Lung_Opacity-4277.png,True,G07044,train,360f399709147c02f28663b77a97523a,d52bd30b1d495e15
Lung_Opacity-4278.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4278.png,Lung_Opacity/masks/Lung_Opacity-4278.png,True,G07045,train,79996f3b96af07b39ba85ac25710a21c,95791e061a7be878
Lung_Opacity-4279.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4279.png,Lung_Opacity/masks/Lung_Opacity-4279.png,True,G07046,train,1d97918cf982e7a0bb445caad65cd51f,dc2e034d4c4e6e6e
Lung_Opacity-428.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-428.png,Lung_Opacity/masks/Lung_Opacity-428.png,True,G07047,test,68c27a4ed72188d141c65d61979b48f2,817947041e54dddf
Lung_Opacity-4280.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4280.png,Lung_Opacity/masks/Lung_Opacity-4280.png,True,G07048,train,e1cfcf7625b3c55382a7dee4f9507d1e,cd2ed2434f1e494d
Lung_Opacity-4281.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4281.png,Lung_Opacity/masks/Lung_Opacity-4281.png,True,G07049,test,8e07c5423d501a7f8b7439093c323174,942c47479f1a2b1f
Lung_Opacity-4282.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4282.png,Lung_Opacity/masks/Lung_Opacity-4282.png,True,G07050,val,d12bb6d61337c5e82c238029c0b6fe5f,dd2a534b2c2d3d26
Lung_Opacity-4283.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4283.png,Lung_Opacity/masks/Lung_Opacity-4283.png,True,G07051,test,e40ff410be1fd6875a6fe89d1cb10075,dc2d425b0d683ef8
Lung_Opacity-4284.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4284.png,Lung_Opacity/masks/Lung_Opacity-4284.png,True,G07052,train,17ed5fb322083a99eb59fd01bf5b0e2c,c03c8f0f0d6e2d6e
Lung_Opacity-4285.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4285.png,Lung_Opacity/masks/Lung_Opacity-4285.png,True,G07053,train,4a5be1b62c5620db89087ba5b8ccd15c,d72e2b9199d1d0d4
Lung_Opacity-4286.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4286.png,Lung_Opacity/masks/Lung_Opacity-4286.png,True,G07054,train,6926ed84ef677ba542d4687ee70915d1,c03d57d23dc82f98
Lung_Opacity-4287.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4287.png,Lung_Opacity/masks/Lung_Opacity-4287.png,True,G07055,train,e59285f6a45a1072a73c72fb102701b1,d03d43431e2c2dfd
Lung_Opacity-4288.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4288.png,Lung_Opacity/masks/Lung_Opacity-4288.png,True,G07056,train,d262698633e64613785216631f3907a1,dd2c470f1e1c4d4d
Lung_Opacity-4289.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4289.png,Lung_Opacity/masks/Lung_Opacity-4289.png,True,G07057,train,b37c57491dc3c54c49f224e0ed095ee4,d939c7125e5e185a
Lung_Opacity-429.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-429.png,Lung_Opacity/masks/Lung_Opacity-429.png,True,G07058,train,2e153e00d7a195ae302811546e4bc285,c03d4a0d3fb77262
Lung_Opacity-4290.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4290.png,Lung_Opacity/masks/Lung_Opacity-4290.png,True,G07059,train,d9f2b1f2e9a9c4506f184e71c9f0c56d,c52a041e1f5d5f5a
Lung_Opacity-4291.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4291.png,Lung_Opacity/masks/Lung_Opacity-4291.png,True,G07060,train,6f88041935d4d47a542ea9ae22b897f3,d2699ec03f63d08e
Lung_Opacity-4292.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4292.png,Lung_Opacity/masks/Lung_Opacity-4292.png,True,G07061,val,b0536cb3c0af742c90e0314e824861bf,886e824b4f4e2f5b
Lung_Opacity-4293.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4293.png,Lung_Opacity/masks/Lung_Opacity-4293.png,True,G07062,train,b81f7263dd7cf971c4b9bd38c9af9886,802847d71fcd4f1d
Lung_Opacity-4294.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4294.png,Lung_Opacity/masks/Lung_Opacity-4294.png,True,G07063,val,36d5bcbe64344da96ef339634ec44659,d02dc20b0e5b2f6f
Lung_Opacity-4295.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4295.png,Lung_Opacity/masks/Lung_Opacity-4295.png,True,G07064,test,f4322b638d144e960450eb175194d226,807fc75e4a4b1a5a
Lung_Opacity-4296.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4296.png,Lung_Opacity/masks/Lung_Opacity-4296.png,True,G07065,train,7d5f26631e209b61aff19e05ccf2ee9e,997b961a495a494f
Lung_Opacity-4297.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4297.png,Lung_Opacity/masks/Lung_Opacity-4297.png,True,G07066,train,8bc66d52b57c8dd4bcb070a9f49529c8,8128175e5f785979
Lung_Opacity-4298.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4298.png,Lung_Opacity/masks/Lung_Opacity-4298.png,True,G07067,test,dc048c758067c3dae7ea3b7fda2c5fab,8b64669b63c33ccc
Lung_Opacity-4299.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4299.png,Lung_Opacity/masks/Lung_Opacity-4299.png,True,G07068,test,0f55a1f55f64311843815192fbc0a73c,cc2dc30f0e2d3dac
Lung_Opacity-43.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-43.png,Lung_Opacity/masks/Lung_Opacity-43.png,True,G07069,train,38b9d62e8c9a1625d9a90d5d2a4b70fb,d03c874f1b1a477c
Lung_Opacity-430.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-430.png,Lung_Opacity/masks/Lung_Opacity-430.png,True,G07070,train,41f66cb0b0ff4c020dec109328c34756,d948c65a1d4d39ad
Lung_Opacity-4300.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4300.png,Lung_Opacity/masks/Lung_Opacity-4300.png,True,G07071,train,5d7aaf983a0e116e4110ef930d867525,dc0d4a0e0cbc95f7
Lung_Opacity-4301.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4301.png,Lung_Opacity/masks/Lung_Opacity-4301.png,True,G07072,train,c3699ab065176af98c3f40828ca0d7f9,817d0c1c384f7f59
Lung_Opacity-4302.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4302.png,Lung_Opacity/masks/Lung_Opacity-4302.png,True,G07073,train,89834f0bc0a16ca30242c116bcdfe440,d47c860e0e0e6f6e
Lung_Opacity-4303.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4303.png,Lung_Opacity/masks/Lung_Opacity-4303.png,True,G07074,train,094d71e46c03fd193eac4377ac5d53d9,817a165f7b2aa8f0
Lung_Opacity-4304.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4304.png,Lung_Opacity/masks/Lung_Opacity-4304.png,True,G07075,train,fe804d57c30579033d73fc3276767c07,81725737392d2d39
Lung_Opacity-4305.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4305.png,Lung_Opacity/masks/Lung_Opacity-4305.png,True,G07076,train,e821840af80af95b289a39ed6ff40c34,897d86161a4f0f3d
Lung_Opacity-4306.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4306.png,Lung_Opacity/masks/Lung_Opacity-4306.png,True,G07077,test,80c354a185c6ebe047d3bba3278042a5,d02b5e0c35233f37
Lung_Opacity-4307.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4307.png,Lung_Opacity/masks/Lung_Opacity-4307.png,True,G07078,train,4320fed614d019d3eeb1ddef45b3633c,dc3c43431f5f581a
Lung_Opacity-4308.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4308.png,Lung_Opacity/masks/Lung_Opacity-4308.png,True,G07079,train,ffaea553933bf3f751b2703c9858a0d8,dc3dc31f002f3d92
Lung_Opacity-4309.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4309.png,Lung_Opacity/masks/Lung_Opacity-4309.png,True,G07080,train,67aca8d0f86a54c0fbf856b8b9c7147d,dc0e831d0d1cfe5c
Lung_Opacity-431.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-431.png,Lung_Opacity/masks/Lung_Opacity-431.png,True,G07081,train,eed458fdfe0b192335121714019575bc,d03f475719485f0d
Lung_Opacity-4310.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4310.png,Lung_Opacity/masks/Lung_Opacity-4310.png,True,G07082,train,b1d3ccbcd3d7f543abe61768120c63f3,c93f52111d0e795d
Lung_Opacity-4311.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4311.png,Lung_Opacity/masks/Lung_Opacity-4311.png,True,G07083,train,18460197d60aa838f918873f179ef97a,d12b5746de19390b
Lung_Opacity-4312.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4312.png,Lung_Opacity/masks/Lung_Opacity-4312.png,True,G07084,val,1057a1c06fa8ca4eb1c9aaedf2bda378,d53c07071e0e6f2d
Lung_Opacity-4313.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4313.png,Lung_Opacity/masks/Lung_Opacity-4313.png,True,G07085,val,eb666073b0691c405dc27a85fbe29c6a,942e43411f4f5e5e
Lung_Opacity-4314.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4314.png,Lung_Opacity/masks/Lung_Opacity-4314.png,True,G07086,val,d19d9fb926d9df5a114d999d695e812e,d42f3391ecc62b91
Lung_Opacity-4315.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4315.png,Lung_Opacity/masks/Lung_Opacity-4315.png,True,G07087,test,268f1d885ae1186d45613df3ae6770e1,917ace1e585a5959
Lung_Opacity-4316.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4316.png,Lung_Opacity/masks/Lung_Opacity-4316.png,True,G07088,train,ced636d51a10377f0f34188152724727,916a165e4d585d3d
Lung_Opacity-4317.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4317.png,Lung_Opacity/masks/Lung_Opacity-4317.png,True,G07089,test,57d0cf527d9a7009f9b190665968aab5,d53f434d4a0d0e2f
Lung_Opacity-4318.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4318.png,Lung_Opacity/masks/Lung_Opacity-4318.png,True,G07090,train,cb0d372db143818458d13f7c64608f5c,d029d253574a5f59
Lung_Opacity-4319.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4319.png,Lung_Opacity/masks/Lung_Opacity-4319.png,True,G07091,val,dc0d3dc39e13d0598cc09e0e7762aefc,d56d474a0b0b0f2f
Lung_Opacity-432.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-432.png,Lung_Opacity/masks/Lung_Opacity-432.png,True,G07092,val,dbc6181b7bccff28d43044611beb65dc,d43bd30b5e2a1e1c
Lung_Opacity-4320.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4320.png,Lung_Opacity/masks/Lung_Opacity-4320.png,True,G07093,test,246c9b28b26a3ca14c4700d148c5fb42,903d4f4e182d7d71
Lung_Opacity-4321.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4321.png,Lung_Opacity/masks/Lung_Opacity-4321.png,True,G07094,train,e7efcf0d7bb1c713535a5844d60eab67,dc2c53430f4e4f0f
Lung_Opacity-4322.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4322.png,Lung_Opacity/masks/Lung_Opacity-4322.png,True,G07095,train,7eb741706a5c21632ba40ae6a7253d30,917ad05a3b1b9a3a
Lung_Opacity-4323.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4323.png,Lung_Opacity/masks/Lung_Opacity-4323.png,True,G07096,train,469cd1cbecf82e1e6787c14eb3ec72f8,dd7f1612165a1939
Lung_Opacity-4324.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4324.png,Lung_Opacity/masks/Lung_Opacity-4324.png,True,G07097,test,e913710052e9715eedcb84bf9757fa5a,8c6c045e5d6d7ce8
Lung_Opacity-4325.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4325.png,Lung_Opacity/masks/Lung_Opacity-4325.png,True,G07098,train,b638317f631cc758f19cd9a25f370fec,d46e03431b5f1a5e
Lung_Opacity-4326.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4326.png,Lung_Opacity/masks/Lung_Opacity-4326.png,True,G07099,train,dbd05f4dd0dbdb01346966f608dc0791,816e064f191d7d6d
Lung_Opacity-4327.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4327.png,Lung_Opacity/masks/Lung_Opacity-4327.png,True,G07100,test,bc5b32f5f1ef39fbceea25a816f9a972,917bc75c596b2c60
Lung_Opacity-4328.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4328.png,Lung_Opacity/masks/Lung_Opacity-4328.png,True,G07101,test,9d50cd0824d135d9765f32d1c53037d0,9d281f1a597969e1
Lung_Opacity-4329.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4329.png,Lung_Opacity/masks/Lung_Opacity-4329.png,True,G07102,train,3c73d169fa9a4343530f8259eb0cc777,b969c71e18591d1b
Lung_Opacity-433.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-433.png,Lung_Opacity/masks/Lung_Opacity-433.png,True,G07103,train,04096d6cfa146e91330f9b43df8208a7,c03df2830cbce9ad
Lung_Opacity-4330.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4330.png,Lung_Opacity/masks/Lung_Opacity-4330.png,True,G07104,train,d8bec4ba4c84dc09dce8a026baf3c17b,d42e05511f2e3e5e
Lung_Opacity-4331.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4331.png,Lung_Opacity/masks/Lung_Opacity-4331.png,True,G07105,train,a7a094f1412280ece6f89a8098a55b2c,916e061c5d7b784e
Lung_Opacity-4332.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4332.png,Lung_Opacity/masks/Lung_Opacity-4332.png,True,G07106,train,0f107c607eb4d634a9f00141996de640,c12b1586d7111f5f
Lung_Opacity-4333.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4333.png,Lung_Opacity/masks/Lung_Opacity-4333.png,True,G07107,train,d33b52482aa9df048a489df0231376c4,893c165a7b7e6066
Lung_Opacity-4334.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4334.png,Lung_Opacity/masks/Lung_Opacity-4334.png,True,G07108,train,148ac5648f3ac6854c9f24d93fe05ee3,8d693c165b1d5b49
Lung_Opacity-4335.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4335.png,Lung_Opacity/masks/Lung_Opacity-4335.png,True,G07109,val,2214798af4d991bdb8b8be0cda80eace,895d165e59e9e8a1
Lung_Opacity-4336.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4336.png,Lung_Opacity/masks/Lung_Opacity-4336.png,True,G07110,train,69bc5b96b47a98d10eda82f9027df5a4,d02f811f5d79686a
Lung_Opacity-4337.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4337.png,Lung_Opacity/masks/Lung_Opacity-4337.png,True,G07111,train,08d3e285f316db04af0eca4354b05c5f,c93e174a1c5b69ac
Lung_Opacity-4338.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4338.png,Lung_Opacity/masks/Lung_Opacity-4338.png,True,G07112,train,7fa6d66f489ce7c6f7e7c0d29415e3a6,dd28c3171e470f0f
Lung_Opacity-4339.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4339.png,Lung_Opacity/masks/Lung_Opacity-4339.png,True,G07113,train,b79601e3e0621d4c2e5b3be57910b657,d03b5796165b5878
Lung_Opacity-434.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-434.png,Lung_Opacity/masks/Lung_Opacity-434.png,True,G07114,train,8459af43f5407d9ad33a40274c92d816,c83d4a2c2c2f6fc3
Lung_Opacity-4340.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4340.png,Lung_Opacity/masks/Lung_Opacity-4340.png,True,G07115,train,7641854e2d44962240ca711e11f3f7ac,85385e4e5a196e6d
Lung_Opacity-4341.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4341.png,Lung_Opacity/masks/Lung_Opacity-4341.png,True,G07116,train,b279df8d62ddd3394c35df17467ea7df,cd4ec148f8eae1e1
Lung_Opacity-4342.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4342.png,Lung_Opacity/masks/Lung_Opacity-4342.png,True,G07117,train,de84872c50acf127842ad5ece3acd5e9,9178c746c4d6167b
Lung_Opacity-4343.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4343.png,Lung_Opacity/masks/Lung_Opacity-4343.png,True,G07118,train,dadd3cc8d7f2000c5cc0df934a6d0dad,c129871e1f5e4b3a
Lung_Opacity-4344.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4344.png,Lung_Opacity/masks/Lung_Opacity-4344.png,True,G07119,train,2bd85eaa42869bc6e14555665ff998cf,c02f4e5c3c2eb4b6
Lung_Opacity-4345.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4345.png,Lung_Opacity/masks/Lung_Opacity-4345.png,True,G07120,train,87ca1fa166019feb45b91cf9ab6b143f,d42b554b5f0acb0e
Lung_Opacity-4346.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4346.png,Lung_Opacity/masks/Lung_Opacity-4346.png,True,G07121,train,ddf06c727cdace13b23063420cad3b3e,c43d53534b0e4e2f
Lung_Opacity-4347.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4347.png,Lung_Opacity/masks/Lung_Opacity-4347.png,True,G07122,train,5191411835a24e193157c44e6a89d026,81ab02171f3b1f1f
Lung_Opacity-4348.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4348.png,Lung_Opacity/masks/Lung_Opacity-4348.png,True,G07123,train,47faaa5b10c984775ce307867cb5faba,ca2d4a0f4b6e0e2f
Lung_Opacity-4349.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4349.png,Lung_Opacity/masks/Lung_Opacity-4349.png,True,G07124,train,011e0a87a0831acc0d928fa63d7c3069,dc2e035b2c4c7ac7
Lung_Opacity-435.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-435.png,Lung_Opacity/masks/Lung_Opacity-435.png,True,G07125,val,7e4a4f862d175865d2deb6e7ebaefa4c,d929471e1c687d78
Lung_Opacity-4350.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4350.png,Lung_Opacity/masks/Lung_Opacity-4350.png,True,G07126,val,83793f1f7420d89c80fba148d04c0c71,890c0e1f1b7f5e58
Lung_Opacity-4351.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4351.png,Lung_Opacity/masks/Lung_Opacity-4351.png,True,G07127,val,1af61c6ce5383dea5e1b209c706222bd,885d061f196dfcb4
Lung_Opacity-4352.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4352.png,Lung_Opacity/masks/Lung_Opacity-4352.png,True,G07128,train,f76c3c0fcadd02158dca69280976db25,aa2d4a451d4e6d1f
Lung_Opacity-4353.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4353.png,Lung_Opacity/masks/Lung_Opacity-4353.png,True,G07129,train,156b9b123fabec8cb9b8abefd97bb255,d56b42961e4b1e1e
Lung_Opacity-4354.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4354.png,Lung_Opacity/masks/Lung_Opacity-4354.png,True,G07130,test,a888a08139290fa4a62c8e664dd2e253,913b160a796d6765
Lung_Opacity-4355.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4355.png,Lung_Opacity/masks/Lung_Opacity-4355.png,True,G07131,test,825d3721886b91baa39c5ae2f87deaf2,807d420a5f4f5f0f
Lung_Opacity-4356.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4356.png,Lung_Opacity/masks/Lung_Opacity-4356.png,True,G07132,train,7e348a8a1e7f44bbad16cb3eb8dc5aea,d52e825909171f5f
Lung_Opacity-4357.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4357.png,Lung_Opacity/masks/Lung_Opacity-4357.png,True,G07133,train,a7e8cf03e1f186c7ba8ee9fe9e2e3711,917c87531f0d1d0f
Lung_Opacity-4358.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4358.png,Lung_Opacity/masks/Lung_Opacity-4358.png,True,G07134,train,1df84b99c0bdb6b926b59465ee8dadba,de2dc34b0f0d0e4e
Lung_Opacity-4359.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4359.png,Lung_Opacity/masks/Lung_Opacity-4359.png,True,G07135,train,b1cf7ca427ee128d3e6157c8e8220f78,817b56861e18db1f
Lung_Opacity-436.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-436.png,Lung_Opacity/masks/Lung_Opacity-436.png,True,G07136,train,0db6599f74d92be9104e20fc9dca51e0,c42a590d2e9e99f9
Lung_Opacity-4360.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4360.png,Lung_Opacity/masks/Lung_Opacity-4360.png,True,G07137,train,ea3d1dc71904753eebd8693e4be81c45,d52ab35355555c54
Lung_Opacity-4361.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4361.png,Lung_Opacity/masks/Lung_Opacity-4361.png,True,G07138,train,931fd34783851506ccaaa92e239dbb9a,c42c07ce5f5a7a4a
Lung_Opacity-4362.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4362.png,Lung_Opacity/masks/Lung_Opacity-4362.png,True,G07139,train,64237facb7cc8e0a93663c5697e0a600,952a07171e0f1d5f
Lung_Opacity-4363.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4363.png,Lung_Opacity/masks/Lung_Opacity-4363.png,True,G07140,val,e087b803a667e2e530be5445ef6c1477,c02f5b5e1c899d9c
Lung_Opacity-4364.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4364.png,Lung_Opacity/masks/Lung_Opacity-4364.png,True,G07141,train,83d4b40dc99669846158b8d49c336dd8,9b68841c9e5f1b1b
Lung_Opacity-4365.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4365.png,Lung_Opacity/masks/Lung_Opacity-4365.png,True,G07142,val,07847767b8808f2b0ebfde0c38f8365c,91784f4c1e8e3d95
Lung_Opacity-4366.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4366.png,Lung_Opacity/masks/Lung_Opacity-4366.png,True,G07143,test,72a1d68ee3bb0e0170989840d4ed0bda,d42ad3575e1a1e58
Lung_Opacity-4367.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4367.png,Lung_Opacity/masks/Lung_Opacity-4367.png,True,G07144,test,bfc187e47eaf647904f3a3b53bb46d80,dc3d015a2f426b57
Lung_Opacity-4368.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4368.png,Lung_Opacity/masks/Lung_Opacity-4368.png,True,G07145,train,3f426f4fa112e7068800c8bbd8ea43df,807f061e5f5b5a4a
Lung_Opacity-4369.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4369.png,Lung_Opacity/masks/Lung_Opacity-4369.png,True,G07146,test,755b9c8b94531c8323b9e5fd7a8e2902,dd2a571a1f4b181d
Lung_Opacity-437.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-437.png,Lung_Opacity/masks/Lung_Opacity-437.png,True,G07147,train,5bf1ffef33d2c25856f3ff48dd259c2a,996dc6569ac33b18
Lung_Opacity-4370.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4370.png,Lung_Opacity/masks/Lung_Opacity-4370.png,True,G07148,val,e41457abc8f4137a522d096e17559e90,846e0253cb4f6e6e
Lung_Opacity-4371.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4371.png,Lung_Opacity/masks/Lung_Opacity-4371.png,True,G07149,train,169cba862aa8bdf754e243a63beb8e45,d1210ede38d83adb
Lung_Opacity-4372.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4372.png,Lung_Opacity/masks/Lung_Opacity-4372.png,True,G07150,train,45e9771f24a65a8a4d0ccbe131e0999f,dd2dc24b590e2c7c
Lung_Opacity-4373.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4373.png,Lung_Opacity/masks/Lung_Opacity-4373.png,True,G07151,train,384272d7c4abfe36c374926ba65dde7f,996d861a451e4b3f
Lung_Opacity-4374.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4374.png,Lung_Opacity/masks/Lung_Opacity-4374.png,True,G07152,train,4571cfa02fc0c7750f1fa4f70530d7ae,993d46c21b15cf1b
Lung_Opacity-4375.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4375.png,Lung_Opacity/masks/Lung_Opacity-4375.png,True,G07153,train,78f7eb27ed40db15da01d4b5047ca24b,dd38d20f57085c1f
Lung_Opacity-4376.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4376.png,Lung_Opacity/masks/Lung_Opacity-4376.png,True,G07154,val,56e6723266a9e6d62a4dfd673114a60c,dd2952d74b5d6848
Lung_Opacity-4377.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4377.png,Lung_Opacity/masks/Lung_Opacity-4377.png,True,G07155,train,438d9250c42f2cee2d0812ec06eb4281,817917861a1f6b5b
Lung_Opacity-4378.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4378.png,Lung_Opacity/masks/Lung_Opacity-4378.png,True,G07156,train,5657fbe38d01dc800bc38088faedc8fe,912b470a5d4b7d59
Lung_Opacity-4379.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4379.png,Lung_Opacity/masks/Lung_Opacity-4379.png,True,G07157,train,59fac994917052f8f9fe7ffa82f07b6b,c43d124b4b3c3cb7
Lung_Opacity-438.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-438.png,Lung_Opacity/masks/Lung_Opacity-438.png,True,G07158,train,9bd1faba56520189459dea0e1e83ecd0,d02fcb0c5e5e6c4a
Lung_Opacity-4380.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4380.png,Lung_Opacity/masks/Lung_Opacity-4380.png,True,G07159,train,6df7d69fb5140c185bf4cc06ed2062df,83281c1f7b787a78
Lung_Opacity-4381.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4381.png,Lung_Opacity/masks/Lung_Opacity-4381.png,True,G07160,train,d49155292c045dea20134faf1713b602,c93c165e5d695878
Lung_Opacity-4382.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4382.png,Lung_Opacity/masks/Lung_Opacity-4382.png,True,G07161,train,b3e243e815a06ed739edeea720afa20e,917a06575a5b5959
Lung_Opacity-4383.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4383.png,Lung_Opacity/masks/Lung_Opacity-4383.png,True,G07162,train,aa2ec3f86385f48dbe8e93eaf8bf4812,986f061e095d1b5f
Lung_Opacity-4384.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4384.png,Lung_Opacity/masks/Lung_Opacity-4384.png,True,G07163,train,e4a95729ca39fb067b66dc9803036341,9d6e405bc31f4947
Lung_Opacity-4385.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4385.png,Lung_Opacity/masks/Lung_Opacity-4385.png,True,G07164,val,e91c304595763610ad2fd61f25bbd781,c129e786be0b4b35
Lung_Opacity-4386.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4386.png,Lung_Opacity/masks/Lung_Opacity-4386.png,True,G07165,val,101b9be6d4dd90734c8922e9834d22a5,d03d43171b3f2a2d
Lung_Opacity-4387.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4387.png,Lung_Opacity/masks/Lung_Opacity-4387.png,True,G07166,test,67e2c4c7cb2f3e4a755c851b23a5811d,913b501f5b593939
Lung_Opacity-4388.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4388.png,Lung_Opacity/masks/Lung_Opacity-4388.png,True,G07167,test,be6622f45b197ac700949d17dfab51cb,817f84161a1b1f5f
Lung_Opacity-4389.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4389.png,Lung_Opacity/masks/Lung_Opacity-4389.png,True,G07168,test,54ccb9d18f92021bf176eccaba9cbf56,d37b04161c1f1f1b
Lung_Opacity-439.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-439.png,Lung_Opacity/masks/Lung_Opacity-439.png,True,G07169,train,03dd0a139856960a804d563f3e0d906d,c0355b5b4a283b5f
Lung_Opacity-4390.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4390.png,Lung_Opacity/masks/Lung_Opacity-4390.png,True,G07170,train,bcc9b3bb3bc0f41ca7edd17b80661e12,817c061e5f5d7958
Lung_Opacity-4391.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4391.png,Lung_Opacity/masks/Lung_Opacity-4391.png,True,G07171,train,03500df1f61ce51e526f1364439c3b03,897d0e19794c7e68
Lung_Opacity-4392.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4392.png,Lung_Opacity/masks/Lung_Opacity-4392.png,True,G07172,train,6c7a322dc18a60a78bbdfd374c38358e,81315e5a784b7b72
Lung_Opacity-4393.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4393.png,Lung_Opacity/masks/Lung_Opacity-4393.png,True,G07173,train,6f08ed4cbc7d87fa415732ad6c938d0c,9c2a131e083eaf3f
Lung_Opacity-4394.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4394.png,Lung_Opacity/masks/Lung_Opacity-4394.png,True,G07174,train,db9aa2879c7f8f1941a3d2daed0c3a99,dd2a530f682dd01f
Lung_Opacity-4395.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4395.png,Lung_Opacity/masks/Lung_Opacity-4395.png,True,G07175,train,b91db6129ddb269d9734c6adcf381064,957b4b4f1d0d1839
Lung_Opacity-4396.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4396.png,Lung_Opacity/masks/Lung_Opacity-4396.png,True,G07176,train,fa09acab03d052af32e5882637b04500,dd79d2165f0a1a0b
Lung_Opacity-4397.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4397.png,Lung_Opacity/masks/Lung_Opacity-4397.png,True,G07177,train,b9a2a02aef4f51904ec6112426ed0f06,987bc69cc8e1b4a5
Lung_Opacity-4398.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4398.png,Lung_Opacity/masks/Lung_Opacity-4398.png,True,G07178,train,7a2273b94558e5f87839cf3baef747a5,817b460a1a5b5f3b
Lung_Opacity-4399.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4399.png,Lung_Opacity/masks/Lung_Opacity-4399.png,True,G07179,test,d37114e3b1223ca61a8da9d2eb2925aa,df2ac00f4e2e13b9
Lung_Opacity-44.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-44.png,Lung_Opacity/masks/Lung_Opacity-44.png,True,G07180,train,8a585c7d9c61bd18e3524984d09282db,d42955c3d71b585c
Lung_Opacity-440.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-440.png,Lung_Opacity/masks/Lung_Opacity-440.png,True,G07181,train,424db17ea55d41bc73116dbfb0b5e52c,98295f593df8a1a1
Lung_Opacity-4400.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4400.png,Lung_Opacity/masks/Lung_Opacity-4400.png,True,G07182,train,0d458f5564528cd3137e332e1300ecb5,993e865beea4a4a4
Lung_Opacity-4401.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4401.png,Lung_Opacity/masks/Lung_Opacity-4401.png,True,G07183,train,89b5c4b974b6c63a5614285c55d08a8a,9928165a7969e9e3
Lung_Opacity-4402.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4402.png,Lung_Opacity/masks/Lung_Opacity-4402.png,True,G07184,val,9f4530a32885310ed6ede16311bda23f,983b522869abefe0
Lung_Opacity-4403.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4403.png,Lung_Opacity/masks/Lung_Opacity-4403.png,True,G07185,val,0d616a01ec82f58326ab0e750140647d,dd38d2131d1d1e1e
Lung_Opacity-4404.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4404.png,Lung_Opacity/masks/Lung_Opacity-4404.png,True,G07186,val,9c0dd59aa1e93927ffde386daa8bf0f3,d07b421a0f2d3f35
Lung_Opacity-4405.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4405.png,Lung_Opacity/masks/Lung_Opacity-4405.png,True,G07187,test,e44f0cf0c1fed70838f3377ac4e6b137,d12b7b5859595919
Lung_Opacity-4406.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4406.png,Lung_Opacity/masks/Lung_Opacity-4406.png,True,G07188,test,81722c10975dd4fe013dc510ae4a6f2c,dc2dd31f4a0f0f09
Lung_Opacity-4407.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4407.png,Lung_Opacity/masks/Lung_Opacity-4407.png,True,G07189,train,fd6eadf2db66f728d6651b7a18ca823a,956a0a1f171f1e59
Lung_Opacity-4408.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4408.png,Lung_Opacity/masks/Lung_Opacity-4408.png,True,G07190,train,c2a462386939c2db3dfe26a6cfe567f6,817e94103f35393f
Lung_Opacity-4409.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4409.png,Lung_Opacity/masks/Lung_Opacity-4409.png,True,G07191,test,1345d26cac19293296f953fa338dbcc1,ce2addc8556a5d0a
Lung_Opacity-441.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-441.png,Lung_Opacity/masks/Lung_Opacity-441.png,True,G07192,train,f55f2ad28db7b02e1b87af0559311aef,cd2a430f6c6c92ed
Lung_Opacity-4410.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4410.png,Lung_Opacity/masks/Lung_Opacity-4410.png,True,G07193,train,bd13e7ed68ec3464100d7c60feda387d,836a65d77a197a03
Lung_Opacity-4411.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4411.png,Lung_Opacity/masks/Lung_Opacity-4411.png,True,G07194,train,b1b2ec3e819f426ae49db1c4fc662734,c52f2e1e140e4f4f
Lung_Opacity-4412.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4412.png,Lung_Opacity/masks/Lung_Opacity-4412.png,True,G07195,train,f296a2c572366c92b7d280e0a49d440d,953e54433ee23ee0
Lung_Opacity-4413.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4413.png,Lung_Opacity/masks/Lung_Opacity-4413.png,True,G07196,train,cb468105a5f75c526ef223e6487f9e0d,9c6f831b4d1c4acb
Lung_Opacity-4414.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4414.png,Lung_Opacity/masks/Lung_Opacity-4414.png,True,G07197,train,18f34d79c7d307f49fdb4d2211ea3331,9d68ae5a1f1a5a1c
Lung_Opacity-4415.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4415.png,Lung_Opacity/masks/Lung_Opacity-4415.png,True,G07198,train,6663cc47a5c31af750e182f387137169,897b961a582d7a39
Lung_Opacity-4416.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4416.png,Lung_Opacity/masks/Lung_Opacity-4416.png,True,G07199,train,9ae08583890475c6aebdd9ac8a92716a,dd2b931e1d181d1e
Lung_Opacity-4417.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4417.png,Lung_Opacity/masks/Lung_Opacity-4417.png,True,G07200,train,4da2599b348fc09703d72af89af1271e,9919061b597d386f
Lung_Opacity-4418.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4418.png,Lung_Opacity/masks/Lung_Opacity-4418.png,True,G07201,train,4f230a05298fac5179c81f96d91e74c1,d42d12131f1e6f3c
Lung_Opacity-4419.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4419.png,Lung_Opacity/masks/Lung_Opacity-4419.png,True,G07202,train,ed3024d9e05d8f87dcea63385c80383e,91691e186b797969
Lung_Opacity-442.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-442.png,Lung_Opacity/masks/Lung_Opacity-442.png,True,G07203,train,8cba70d06d01f69d181f2895f41b2cf7,887fe2d71e4c085d
Lung_Opacity-4420.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4420.png,Lung_Opacity/masks/Lung_Opacity-4420.png,True,G07204,val,2d3665884eb934b7e3fd1d594b239fd0,997946561a595f1a
Lung_Opacity-4421.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4421.png,Lung_Opacity/masks/Lung_Opacity-4421.png,True,G07205,val,27757fae81b859d05b373351d9381f05,c42f554b2f2e2c0f
Lung_Opacity-4422.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4422.png,Lung_Opacity/masks/Lung_Opacity-4422.png,True,G07206,val,9e5851f9a4c51baea7e8b275fac7dc4f,d87b0587561a4ddc
Lung_Opacity-4423.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4423.png,Lung_Opacity/masks/Lung_Opacity-4423.png,True,G07207,test,89520a29c1a670745aa29d2f6611add3,c17b560a5f4c5e0e
Lung_Opacity-4424.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4424.png,Lung_Opacity/masks/Lung_Opacity-4424.png,True,G07208,test,c97d88d016533147f1571fbeb16ef8b1,c46f434f452f26a3
Lung_Opacity-4425.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4425.png,Lung_Opacity/masks/Lung_Opacity-4425.png,True,G07209,train,9aee13e2fc467aa33c98593f77ad6141,887845171bcd4f1f
Lung_Opacity-4426.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4426.png,Lung_Opacity/masks/Lung_Opacity-4426.png,True,G07210,train,207862775c21c4d5e3d188a8774f4e56,ce19532e3e352565
Lung_Opacity-4427.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4427.png,Lung_Opacity/masks/Lung_Opacity-4427.png,True,G07211,train,6b0129fd84710f261120f83e1f5ca221,9178085d1b5f3a5d
Lung_Opacity-4428.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4428.png,Lung_Opacity/masks/Lung_Opacity-4428.png,True,G07212,train,f7da7ac024195c3ff2dd8df5ab1afaef,c02d121f187d5e7e
Lung_Opacity-4429.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4429.png,Lung_Opacity/masks/Lung_Opacity-4429.png,True,G07213,train,37e8712d574ffc202025dbfa7c0f85c6,c03f521b0e2d7e2d
Lung_Opacity-443.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-443.png,Lung_Opacity/masks/Lung_Opacity-443.png,True,G07214,train,c212cfd70c37eae3288ea22e9f77bf15,9c6e861e4f1d0c4f
Lung_Opacity-4430.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4430.png,Lung_Opacity/masks/Lung_Opacity-4430.png,True,G07215,train,f2247987d005fcfedf0d66ce819e833c,d4af1243471f0b1f
Lung_Opacity-4431.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4431.png,Lung_Opacity/masks/Lung_Opacity-4431.png,True,G07216,train,9a9e0177c53a81a1bf460aa6a7dbe95d,c46f074e0ded21e5
Lung_Opacity-4432.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4432.png,Lung_Opacity/masks/Lung_Opacity-4432.png,True,G07217,train,c8e4f3c0647c7ccae2e2c557ce71dd05,817b14865b1d1d5f
Lung_Opacity-4433.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4433.png,Lung_Opacity/masks/Lung_Opacity-4433.png,True,G07218,train,883f46d59cce66e8e472c4f1461d5b9d,897d05861e871f9e
Lung_Opacity-4434.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4434.png,Lung_Opacity/masks/Lung_Opacity-4434.png,True,G07219,train,f79c38aeabba9892476308479173bbd3,cd3f1647164a2d4e
Lung_Opacity-4435.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4435.png,Lung_Opacity/masks/Lung_Opacity-4435.png,True,G07220,train,67b82de13132fefd3d5f11251754fb2b,d13a921b3c385b3e
Lung_Opacity-4436.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4436.png,Lung_Opacity/masks/Lung_Opacity-4436.png,True,G07221,train,fefc54cb5321603e0d801e6e829be04f,d43d420b0e6f2d4f
Lung_Opacity-4437.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4437.png,Lung_Opacity/masks/Lung_Opacity-4437.png,True,G07222,train,3685c4d866fc298f70c3ef22ada61c92,c03d471a1b4f3d4e
Lung_Opacity-4438.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4438.png,Lung_Opacity/masks/Lung_Opacity-4438.png,True,G07223,val,eb31460f6a6912a2703ffe0d728727bc,9839c38696cacd9b
Lung_Opacity-4439.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4439.png,Lung_Opacity/masks/Lung_Opacity-4439.png,True,G07224,val,c5ef3d64c91b8145fefe95ec585cfd21,d42ec74e4e2a6a2b
Lung_Opacity-444.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-444.png,Lung_Opacity/masks/Lung_Opacity-444.png,True,G07225,test,723442f127e36b22fe059c6d3e10f421,817a041e1f5b5f5a
Lung_Opacity-4440.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4440.png,Lung_Opacity/masks/Lung_Opacity-4440.png,True,G07226,test,523bad0e30c902916b46a00f979b8baf,917e82191d7979f0
Lung_Opacity-4441.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4441.png,Lung_Opacity/masks/Lung_Opacity-4441.png,True,G07227,val,73b287816ec24d8d37461474ddee7954,dc7d420f0c0f2c6f
Lung_Opacity-4442.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4442.png,Lung_Opacity/masks/Lung_Opacity-4442.png,True,G07228,train,242ad58a5defcaab87c5d5bb91b79a00,9139561a3b7b2939
Lung_Opacity-4443.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4443.png,Lung_Opacity/masks/Lung_Opacity-4443.png,True,G07229,test,60408fb91ec699a9363e090e33f1ff3c,d03d47130f4f0d4f
Lung_Opacity-4444.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4444.png,Lung_Opacity/masks/Lung_Opacity-4444.png,True,G07230,train,75e4e0dd22fc6a9e8281649e30c07d16,d82d0f3866723727
Lung_Opacity-4445.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4445.png,Lung_Opacity/masks/Lung_Opacity-4445.png,True,G07231,train,0041a5c2281ce54f3e4840b166733a9c,817917061c5e5f3d
Lung_Opacity-4446.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4446.png,Lung_Opacity/masks/Lung_Opacity-4446.png,True,G07232,train,6053be1b9d1abd4e0cdb53d8280a2bcd,81295e5f1b391b0f
Lung_Opacity-4447.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4447.png,Lung_Opacity/masks/Lung_Opacity-4447.png,True,G07233,train,572d186a10dc9bbc5096f3f61678ead5,946cc61a5be51e69
Lung_Opacity-4448.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4448.png,Lung_Opacity/masks/Lung_Opacity-4448.png,True,G07234,train,6d9c3272e4f02bc5e6564803cf8b5f6e,c43dc3535f0e5a4a
Lung_Opacity-4449.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4449.png,Lung_Opacity/masks/Lung_Opacity-4449.png,True,G07235,train,37bfebcc1e947f44f34bd15cb58797d6,9dec8712451e0f4f
Lung_Opacity-445.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-445.png,Lung_Opacity/masks/Lung_Opacity-445.png,True,G07236,train,68a131ac799366a70465f05c0d2dbb61,cc3cc30f400d7e6f
Lung_Opacity-4450.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4450.png,Lung_Opacity/masks/Lung_Opacity-4450.png,True,G07237,test,f623c253a93f0e742fbbfbf11b4a32a2,d13b5a90da197b29
Lung_Opacity-4451.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4451.png,Lung_Opacity/masks/Lung_Opacity-4451.png,True,G07238,train,832feff704daca701b9c162e419ec94c,d42a131b2c3a7377
Lung_Opacity-4452.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4452.png,Lung_Opacity/masks/Lung_Opacity-4452.png,True,G07239,train,abd749fdaa5432ff97ed08abbe185e02,907e87971a1e0f0f
Lung_Opacity-4453.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4453.png,Lung_Opacity/masks/Lung_Opacity-4453.png,True,G07240,train,69b3178eb420a5d4c191b86ebc55ebc6,de2d410b0a2e2ebf
Lung_Opacity-4454.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4454.png,Lung_Opacity/masks/Lung_Opacity-4454.png,True,G07241,val,407e4723c7fa64ef975f67f31d0a7c3b,d76c0f2b3d703033
Lung_Opacity-4455.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4455.png,Lung_Opacity/masks/Lung_Opacity-4455.png,True,G07242,train,6ee64ee613c9020d934587111967db0b,d97d061a1d6b3bc0
Lung_Opacity-4456.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4456.png,Lung_Opacity/masks/Lung_Opacity-4456.png,True,G07243,train,a8f6f686e753fe2b3ddb3adde4817681,907b43185b1b3d3e
Lung_Opacity-4457.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4457.png,Lung_Opacity/masks/Lung_Opacity-4457.png,True,G07244,train,6379a491ec3bdc1aaf376c778f5b7e01,802f450c4e6e4f7f
Lung_Opacity-4458.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4458.png,Lung_Opacity/masks/Lung_Opacity-4458.png,True,G07245,val,15393b183e34a7736bc55b05d278c318,912a845f5e7a7859
Lung_Opacity-4459.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4459.png,Lung_Opacity/masks/Lung_Opacity-4459.png,True,G07246,val,2e506a15c643edf2d6f7f9e4b3d6d6f4,9d7bc0160a4d1fda
Lung_Opacity-446.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-446.png,Lung_Opacity/masks/Lung_Opacity-446.png,True,G07247,val,fea28240aede6dc2323c23484beede61,d07d474b0a6d2e2e
Lung_Opacity-4460.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4460.png,Lung_Opacity/masks/Lung_Opacity-4460.png,True,G07248,test,6c9363e5a6402ba809e25a401bd4e133,862ac34f6ee6b4e0
Lung_Opacity-4461.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4461.png,Lung_Opacity/masks/Lung_Opacity-4461.png,True,G07249,test,13e9e320dd4551e52ac46cfd126c59d4,ff5a1fd0dac0c0c2
Lung_Opacity-4462.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4462.png,Lung_Opacity/masks/Lung_Opacity-4462.png,True,G07250,train,b89020a8ef135bf70d4b7959b75ae96e,c03bd7521f586a69
Lung_Opacity-4463.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4463.png,Lung_Opacity/masks/Lung_Opacity-4463.png,True,G07251,train,24b014c77813beef900200901657a3ee,8d6e0c197870f2f3
Lung_Opacity-4464.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4464.png,Lung_Opacity/masks/Lung_Opacity-4464.png,True,G07252,train,b3f55d5d491ac13a7693e13a646b4cf5,8178165b197deaa5
Lung_Opacity-4465.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4465.png,Lung_Opacity/masks/Lung_Opacity-4465.png,True,G07253,train,071075646c85c5b5f822e7c909966b6a,c76329c18e4eed07
Lung_Opacity-4466.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4466.png,Lung_Opacity/masks/Lung_Opacity-4466.png,True,G07254,train,8df1f9a1942453e312ed2f192aa7084a,d03dd2d35d545c1c
Lung_Opacity-4467.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4467.png,Lung_Opacity/masks/Lung_Opacity-4467.png,True,G07255,train,7df540165d8bf45a29e520c3f391fd8d,885f0f1e5a49cdcc
Lung_Opacity-4468.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4468.png,Lung_Opacity/masks/Lung_Opacity-4468.png,True,G07256,train,8cc6e31a493a1460cf7bdec961d8088f,8179041e1f1b5f1f
Lung_Opacity-4469.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4469.png,Lung_Opacity/masks/Lung_Opacity-4469.png,True,G07257,train,653b034924f59cf56cffd09472a73f20,90191a1bf8e8e5e7
Lung_Opacity-447.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-447.png,Lung_Opacity/masks/Lung_Opacity-447.png,True,G07258,train,bc4541466f7ac6e0950f32f675cd502b,8138164f7979ea69
Lung_Opacity-4470.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4470.png,Lung_Opacity/masks/Lung_Opacity-4470.png,True,G07259,val,2f78b5beb9748c6b48d1f14cb02e4f5f,d02dd2d25b4bda4a
Lung_Opacity-4471.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4471.png,Lung_Opacity/masks/Lung_Opacity-4471.png,True,G07260,test,acfa0e6d576e341fd51b456ad77a612d,d03d4f0a081f1fdb
Lung_Opacity-4472.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4472.png,Lung_Opacity/masks/Lung_Opacity-4472.png,True,G07261,train,7f89f7ecd3c68c1e7c21c4dd6320345f,d23d57020b06ff0f
Lung_Opacity-4473.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4473.png,Lung_Opacity/masks/Lung_Opacity-4473.png,True,G07262,train,207dd48b71f9b7fecd29d13977a8c15d,983fd70a1c191b3d
Lung_Opacity-4474.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4474.png,Lung_Opacity/masks/Lung_Opacity-4474.png,True,G07263,train,9b548fc46f7284302345f5bf9084ba9a,d03f420b4a2b7f2d
Lung_Opacity-4475.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4475.png,Lung_Opacity/masks/Lung_Opacity-4475.png,True,G07264,train,4d5d0ed805c69e796ee1ac74a59bff5e,d02ce7921fc12f9c
Lung_Opacity-4476.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4476.png,Lung_Opacity/masks/Lung_Opacity-4476.png,True,G07265,train,6ad00c5c3d7fff981b8e982a9306fc4f,c16816135b79f86b
Lung_Opacity-4477.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4477.png,Lung_Opacity/masks/Lung_Opacity-4477.png,True,G07266,val,2463e79fef1acd8e4c5dd406a0eab02e,d42e054e6e6ceb61
Lung_Opacity-4478.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4478.png,Lung_Opacity/masks/Lung_Opacity-4478.png,True,G07267,val,768e80c436bead28b71668d353cda3aa,9c4b430f49494f6f
Lung_Opacity-4479.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4479.png,Lung_Opacity/masks/Lung_Opacity-4479.png,True,G07268,test,89e64fd35db38e7201c9fb66ae8896c7,883d02125b6d7dfc
Lung_Opacity-448.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-448.png,Lung_Opacity/masks/Lung_Opacity-448.png,True,G07269,train,aa7dfd454d46059a803982b5a9dec448,887d071a4d6d38e7
Lung_Opacity-4480.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4480.png,Lung_Opacity/masks/Lung_Opacity-4480.png,True,G07270,test,65845b1af72d6c850f913184ef2961a9,81780a1ffbe9a3a2
Lung_Opacity-4481.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4481.png,Lung_Opacity/masks/Lung_Opacity-4481.png,True,G07271,train,172a4fd1a2490321bd2a802011096db2,d42d071e693d3c6c
Lung_Opacity-4482.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4482.png,Lung_Opacity/masks/Lung_Opacity-4482.png,True,G07272,train,a0b27788212ad4a719150fdeb11950f6,d52b1e1e5c0b5b0d
Lung_Opacity-4483.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4483.png,Lung_Opacity/masks/Lung_Opacity-4483.png,True,G07273,train,efd2983b3ca9863505d0bbdc794b9580,803b07565e5f585e
Lung_Opacity-4484.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4484.png,Lung_Opacity/masks/Lung_Opacity-4484.png,True,G07274,train,8619d8da5a43020cac30f2b6e22eb953,d835d317aa45bb41
Lung_Opacity-4485.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4485.png,Lung_Opacity/masks/Lung_Opacity-4485.png,True,G07275,train,3dc351a6ef26be10d59a30db56c599b1,d92b12091d597d5b
Lung_Opacity-4486.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4486.png,Lung_Opacity/masks/Lung_Opacity-4486.png,True,G07276,train,b0ec45844f202757dc7b2fea066b09c8,842e070f5d1c7f4d
Lung_Opacity-4487.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4487.png,Lung_Opacity/masks/Lung_Opacity-4487.png,True,G07277,train,f41317e6b5db7bcb309caee9344d60e5,dd2d06864b1d4d4f
Lung_Opacity-4488.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4488.png,Lung_Opacity/masks/Lung_Opacity-4488.png,True,G07278,train,77c9faf2f2da74663bf5002f28b4e712,cc3d534b1c4a3d4e
Lung_Opacity-4489.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4489.png,Lung_Opacity/masks/Lung_Opacity-4489.png,True,G07279,train,cdc97799719a2bfca74a3d0dcc25c4e2,cd0c930f1d4c5d4f
Lung_Opacity-449.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-449.png,Lung_Opacity/masks/Lung_Opacity-449.png,True,G07280,train,9663e26ea82df836dc5fffb224cee26d,d0af2f450f4e0d4e
Lung_Opacity-4490.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4490.png,Lung_Opacity/masks/Lung_Opacity-4490.png,True,G07281,train,4cebddf14a198a0407f027c67b0ffacd,dc3d43430b0a6f1f
Lung_Opacity-4491.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4491.png,Lung_Opacity/masks/Lung_Opacity-4491.png,True,G07282,train,ea04eb5a0affca9bb250ea32ea8822b3,dc3dc34b0c6c4cbc
Lung_Opacity-4492.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4492.png,Lung_Opacity/masks/Lung_Opacity-4492.png,True,G07283,train,8a549ffbb2ee61091d22ffe8fa11b05d,d034ed43d30baa4f
Lung_Opacity-4493.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4493.png,Lung_Opacity/masks/Lung_Opacity-4493.png,True,G07284,test,8d699e670334af45493d29d8e2e93ae2,984f474b0f584d7c
Lung_Opacity-4494.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4494.png,Lung_Opacity/masks/Lung_Opacity-4494.png,True,G07285,val,4feaf2b55ed0986762e35144ed540447,d86cc7c21d484f5b
Lung_Opacity-4495.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4495.png,Lung_Opacity/masks/Lung_Opacity-4495.png,True,G07286,val,792cc60247d8678a9bf6da40cb104e72,dc3dc35b0b680f1c
Lung_Opacity-4496.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4496.png,Lung_Opacity/masks/Lung_Opacity-4496.png,True,G07287,val,2011b041ba9de082d9e16bcdf02021fb,d53c14c78b1a9e4e
Lung_Opacity-4497.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4497.png,Lung_Opacity/masks/Lung_Opacity-4497.png,True,G07288,test,afd4926949878c55aa27e998f79c6121,dd29c31b0f6d3838
Lung_Opacity-4498.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4498.png,Lung_Opacity/masks/Lung_Opacity-4498.png,True,G07289,test,224dd954ee56a4a205c18281044161fb,d522930f683b333b
Lung_Opacity-4499.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4499.png,Lung_Opacity/masks/Lung_Opacity-4499.png,True,G07290,train,3adf238c92b96169edb07f30106b3ad3,d03d6b02de0aed69
Lung_Opacity-45.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-45.png,Lung_Opacity/masks/Lung_Opacity-45.png,True,G07291,val,3e636e166ae8a40fd69f29d8e28b8789,dd7f421b0c4b2b86
Lung_Opacity-450.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-450.png,Lung_Opacity/masks/Lung_Opacity-450.png,True,G07292,test,176302aeeeb13649b70fd5829a16f0ea,d05b52007b29bbbd
Lung_Opacity-4500.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4500.png,Lung_Opacity/masks/Lung_Opacity-4500.png,True,G07293,train,c1ee1f228602d0243167f76704f12dd1,95691a9e1b3be982
Lung_Opacity-4501.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4501.png,Lung_Opacity/masks/Lung_Opacity-4501.png,True,G07294,train,13f8fa0267787cab4507c77d7a0f27d7,cc2ad25b4b4a5a5e
Lung_Opacity-4502.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4502.png,Lung_Opacity/masks/Lung_Opacity-4502.png,True,G07295,train,4216f6161935847048aca9c1f4e3e628,887f87521a4d781f
Lung_Opacity-4503.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4503.png,Lung_Opacity/masks/Lung_Opacity-4503.png,True,G07296,train,82cfbf5d1a05fb11be12b7705f5895e4,917bc6129894bdad
Lung_Opacity-4504.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4504.png,Lung_Opacity/masks/Lung_Opacity-4504.png,True,G07297,train,8b9ee8e20063788ea8e956ea11d4fbff,c95e125bf829e1e1
Lung_Opacity-4505.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4505.png,Lung_Opacity/masks/Lung_Opacity-4505.png,True,G07298,train,84fe952921608ea5ae6ee83129af75a7,c03fd24b494d7939
Lung_Opacity-4506.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4506.png,Lung_Opacity/masks/Lung_Opacity-4506.png,True,G07299,train,9934927873987d18ed8a8f6856c93c02,d81bd32a3cb4b692
Lung_Opacity-4507.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4507.png,Lung_Opacity/masks/Lung_Opacity-4507.png,True,G07300,val,95f0981a783840a3b0bfe04c3d277308,992a4f5e385960f6
Lung_Opacity-4508.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4508.png,Lung_Opacity/masks/Lung_Opacity-4508.png,True,G07301,train,493e63319e8139bda7fce25dfc1fb96a,885f16181bb92fce
Lung_Opacity-4509.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4509.png,Lung_Opacity/masks/Lung_Opacity-4509.png,True,G07302,test,f056f0c73094595a8c1c5f011a321224,c96f06921ecd4b78
Lung_Opacity-451.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-451.png,Lung_Opacity/masks/Lung_Opacity-451.png,True,G07303,train,16b74b885fb0a8f658ddec1e0f07dd68,887b17525a5de869
Lung_Opacity-4510.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4510.png,Lung_Opacity/masks/Lung_Opacity-4510.png,True,G07304,val,b5e9b7a4c656d4d5103f58c1bae7140f,803f4d5e1a5ae86d
Lung_Opacity-4511.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4511.png,Lung_Opacity/masks/Lung_Opacity-4511.png,True,G07305,train,03d9e0706b08d98862830683cf89fa2f,9978561a4b1e3aab
Lung_Opacity-4512.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4512.png,Lung_Opacity/masks/Lung_Opacity-4512.png,True,G07306,train,e92755253986d9f514bf315cd9a48b1e,987b851e1e1b1b1b
Lung_Opacity-4513.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4513.png,Lung_Opacity/masks/Lung_Opacity-4513.png,True,G07307,train,244c3636a4bf93e46a64bc4b366acc46,817c2e5a7d786865
Lung_Opacity-4514.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4514.png,Lung_Opacity/masks/Lung_Opacity-4514.png,True,G07308,train,d1a18fe3a374720c905e44032d10d161,916806121bcfdfba
Lung_Opacity-4515.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4515.png,Lung_Opacity/masks/Lung_Opacity-4515.png,True,G07309,val,c8e249cac313766075117b46ff48656c,912a85149f5f1e5e
Lung_Opacity-4516.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4516.png,Lung_Opacity/masks/Lung_Opacity-4516.png,True,G07310,test,af976183dbd0016fc15ad3a4bf21c70f,d13b121a1b1b3abb
Lung_Opacity-4517.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4517.png,Lung_Opacity/masks/Lung_Opacity-4517.png,True,G07311,test,8f5765659b67f4db402a0e1283c10e17,887ded859716929a
Lung_Opacity-4518.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4518.png,Lung_Opacity/masks/Lung_Opacity-4518.png,True,G07312,train,7c5baa0f566acec86302370acae002b2,dc2e014f2e2e2b6b
Lung_Opacity-4519.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4519.png,Lung_Opacity/masks/Lung_Opacity-4519.png,True,G07313,train,ac759026813d18fb1c79dc85c76846d4,d42e470b0d2e2daf
Lung_Opacity-452.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-452.png,Lung_Opacity/masks/Lung_Opacity-452.png,True,G07314,val,b383a585cb668a2e9ba20067873f7f69,d42d53534f434e5c
Lung_Opacity-4520.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4520.png,Lung_Opacity/masks/Lung_Opacity-4520.png,True,G07315,train,080e199fc7673b596dcebc93b503eb0b,9129161f5a397b69
Lung_Opacity-4521.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4521.png,Lung_Opacity/masks/Lung_Opacity-4521.png,True,G07316,train,059700d4ce62b2f082e71cdb45dd703f,917b061e495b3a6b
Lung_Opacity-4522.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4522.png,Lung_Opacity/masks/Lung_Opacity-4522.png,True,G07317,train,882d564a181efc1095af9c4cac001f9f,d43ddb03cb4a1a6a
Lung_Opacity-4523.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4523.png,Lung_Opacity/masks/Lung_Opacity-4523.png,True,G07318,train,38ffad0184d632f32fc8080e97fedb9d,c42dd21b18f8cbcd
Lung_Opacity-4524.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4524.png,Lung_Opacity/masks/Lung_Opacity-4524.png,True,G07319,train,395208a0f21f94d4e08b8370f49ea7de,c43d420b0b0fef6d
Lung_Opacity-4525.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4525.png,Lung_Opacity/masks/Lung_Opacity-4525.png,True,G07320,train,beb32268c4f39e4f0a6ddd8ebb6ac875,987f00171e5a5b5b
Lung_Opacity-4526.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4526.png,Lung_Opacity/masks/Lung_Opacity-4526.png,True,G07321,train,de77c6b970eb9b8ea377fb766fab16b7,c03df1528b52eb5a
Lung_Opacity-4527.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4527.png,Lung_Opacity/masks/Lung_Opacity-4527.png,True,G07322,train,812f792f0de5441ac30fdf0a11f5bc06,d53d46c61a1c4f4d
Lung_Opacity-4528.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4528.png,Lung_Opacity/masks/Lung_Opacity-4528.png,True,G07323,train,20501f33859f1f67eb02dd11a206e778,d52ad2179bc24b5a
Lung_Opacity-4529.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4529.png,Lung_Opacity/masks/Lung_Opacity-4529.png,True,G07324,train,8c5f40724bdcae2b911ee335e7cad105,dc6c42531f492d3e
Lung_Opacity-453.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-453.png,Lung_Opacity/masks/Lung_Opacity-453.png,True,G07325,train,2ed6d31ee52a3eb9a56383f8dd5aa853,917b041f5b5b585a
Lung_Opacity-4530.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4530.png,Lung_Opacity/masks/Lung_Opacity-4530.png,True,G07326,train,404dbc75754c68b151a9fb60b79aff01,c839430c387b3f3d
Lung_Opacity-4531.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4531.png,Lung_Opacity/masks/Lung_Opacity-4531.png,True,G07327,val,93bbc6f9c69e1e530f64a8d1602d5dba,946b795c64672726
Lung_Opacity-4532.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4532.png,Lung_Opacity/masks/Lung_Opacity-4532.png,True,G07328,val,3e4d3fa7d7f65445ae8278d13899bc0f,c03d5a5a5b697d09
Lung_Opacity-4533.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4533.png,Lung_Opacity/masks/Lung_Opacity-4533.png,True,G07329,test,9e6100794ca2d3fd5a53222a8593d991,957807430b4f2e5f
Lung_Opacity-4534.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4534.png,Lung_Opacity/masks/Lung_Opacity-4534.png,True,G07330,train,a186e0e993a2468ef4cd24c47f17544c,d13926061f1e5f1d
Lung_Opacity-4535.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4535.png,Lung_Opacity/masks/Lung_Opacity-4535.png,True,G07331,train,04106156f44342a95439699567b2ecd7,cd2ed20b4f280daf
Lung_Opacity-4536.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4536.png,Lung_Opacity/masks/Lung_Opacity-4536.png,True,G07332,test,3041b8771b8ba10fa6260978503dbf16,887d4e5658494f4f
Lung_Opacity-4537.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4537.png,Lung_Opacity/masks/Lung_Opacity-4537.png,True,G07333,test,1a0ee50c7fb10a4299ee05a55e139983,913b4acf38a59339
Lung_Opacity-4538.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4538.png,Lung_Opacity/masks/Lung_Opacity-4538.png,True,G07334,train,1036940775466a68d5e22d7ac5d1e101,c42f194d0c464f7f
Lung_Opacity-4539.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4539.png,Lung_Opacity/masks/Lung_Opacity-4539.png,True,G07335,train,bcddbb1fd741a7c6c2d600a5d8a6587d,846e03c30f4d4e7f
Lung_Opacity-454.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-454.png,Lung_Opacity/masks/Lung_Opacity-454.png,True,G07336,train,0eaf6d82601543ffc1a123597bdba2df,906e87175b4f0d4d
Lung_Opacity-4540.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4540.png,Lung_Opacity/masks/Lung_Opacity-4540.png,True,G07337,train,de7abb02fd3308ece5c7c58ff6a1d5c3,817b05941e1b5b5f
Lung_Opacity-4541.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4541.png,Lung_Opacity/masks/Lung_Opacity-4541.png,True,G07338,train,6233dd14f3dfa8c3a68bf2179a46fad9,913a6b43c3a73e54
Lung_Opacity-4542.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4542.png,Lung_Opacity/masks/Lung_Opacity-4542.png,True,G07339,train,0835f93c768fb20385a66c8a5a144403,827f0415171f1f5d
Lung_Opacity-4543.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4543.png,Lung_Opacity/masks/Lung_Opacity-4543.png,True,G07340,train,b5eaee9f9e9110777b114ecbfe065563,9578081e1b796d5b
Lung_Opacity-4544.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4544.png,Lung_Opacity/masks/Lung_Opacity-4544.png,True,G07341,val,3c1aef7ab75720fd0482e0d4bf474f81,de35420b6e0e2e4f
Lung_Opacity-4545.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4545.png,Lung_Opacity/masks/Lung_Opacity-4545.png,True,G07342,train,ffcec07278b6ce98d05590e4157137bd,883deb828d9da595
Lung_Opacity-4546.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4546.png,Lung_Opacity/masks/Lung_Opacity-4546.png,True,G07343,train,b876196e3c704734d6a5254d4a139605,907a171e3f296565
Lung_Opacity-4547.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4547.png,Lung_Opacity/masks/Lung_Opacity-4547.png,True,G07344,train,a5965efba77743980cbadbd013de17e5,fd2cd21bd44b1d41
Lung_Opacity-4548.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4548.png,Lung_Opacity/masks/Lung_Opacity-4548.png,True,G07345,val,e9acbe8bebd96b3e948eda9d674a2f07,c83d175a191d5f19
Lung_Opacity-4549.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4549.png,Lung_Opacity/masks/Lung_Opacity-4549.png,True,G07346,train,a95221a9f0518adcc47720294b693b50,816817031b5d7f5b
Lung_Opacity-455.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-455.png,Lung_Opacity/masks/Lung_Opacity-455.png,True,G07347,val,68eca9e49417a42e7759e841b9459bf5,984f861a5a595e4f
Lung_Opacity-4550.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4550.png,Lung_Opacity/masks/Lung_Opacity-4550.png,True,G07348,train,0e2e28fc16309b6a1079c5da111a6327,917b861e1a5f383a
Lung_Opacity-4551.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4551.png,Lung_Opacity/masks/Lung_Opacity-4551.png,True,G07349,test,9df328ec59f4dd70ee3ee62e77029d30,80729d15177f7478
Lung_Opacity-4552.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4552.png,Lung_Opacity/masks/Lung_Opacity-4552.png,True,G07350,test,7145f46a4ee062c52021b6120ff9de8e,c83d6b4b082c3f3b
Lung_Opacity-4553.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4553.png,Lung_Opacity/masks/Lung_Opacity-4553.png,True,G07351,train,396253d8776b1f9e42304c418130488f,887e06504f455f5f
Lung_Opacity-4554.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4554.png,Lung_Opacity/masks/Lung_Opacity-4554.png,True,G07352,test,c8f7dcae62476cb05b6c08c841c97802,9128061f7ae879f9
Lung_Opacity-4555.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4555.png,Lung_Opacity/masks/Lung_Opacity-4555.png,True,G07353,train,a4e0f3a8973feb984e9bd4667157edd4,997b05161e495e7a
Lung_Opacity-4556.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4556.png,Lung_Opacity/masks/Lung_Opacity-4556.png,True,G07354,train,e65c33c8dd57cadd86f4a26184552f37,c03d530b1d693d3d
Lung_Opacity-4557.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4557.png,Lung_Opacity/masks/Lung_Opacity-4557.png,True,G07355,train,5f1dafef93f35e75f4ea7cce03f0cf0b,d93b961a1d59191e
Lung_Opacity-4558.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4558.png,Lung_Opacity/masks/Lung_Opacity-4558.png,True,G07356,train,d325d3e6439447bc8f36a38becbc7461,9d38c7929b0bd616
Lung_Opacity-4559.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4559.png,Lung_Opacity/masks/Lung_Opacity-4559.png,True,G07357,train,b18a34a7ce70eedbfe2bb6601c38e183,8039569a5d6b5b59
Lung_Opacity-456.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-456.png,Lung_Opacity/masks/Lung_Opacity-456.png,True,G07358,train,a17bce783491e1d37f1aae49e8264f31,997b861a194959dd
Lung_Opacity-4560.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4560.png,Lung_Opacity/masks/Lung_Opacity-4560.png,True,G07359,train,dcedb9cea04db5b0fc003734cb66614c,953b69e592968969
Lung_Opacity-4561.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4561.png,Lung_Opacity/masks/Lung_Opacity-4561.png,True,G07360,val,065807b037349d293de600160be8ab46,d979d4505855575c
Lung_Opacity-4562.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4562.png,Lung_Opacity/masks/Lung_Opacity-4562.png,True,G07361,train,540865497fe15e017c2b49f37817da72,d52f4a4f4c0c3e3c
Lung_Opacity-4563.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4563.png,Lung_Opacity/masks/Lung_Opacity-4563.png,True,G07362,train,5a33bb5e02cb0e2e92877ea7e55c1e4b,915f941c1a5b1b5e
Lung_Opacity-4564.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4564.png,Lung_Opacity/masks/Lung_Opacity-4564.png,True,G07363,train,1a4c6364fdbcdee48e84d743b18f01b6,d03ce6c3db4a854b
Lung_Opacity-4565.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4565.png,Lung_Opacity/masks/Lung_Opacity-4565.png,True,G07364,train,45270cb6e421d8bee80f21b8d8b326b7,c13d525e5c397929
Lung_Opacity-4566.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4566.png,Lung_Opacity/masks/Lung_Opacity-4566.png,True,G07365,train,623c5faf0d61fd9577f67ec1e7b84ed8,d92f8613071f1e4d
Lung_Opacity-4567.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4567.png,Lung_Opacity/masks/Lung_Opacity-4567.png,True,G07366,train,6c1e4946a0d016ed554478d01889ae23,853e164c5c7d6c4e
Lung_Opacity-4568.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4568.png,Lung_Opacity/masks/Lung_Opacity-4568.png,True,G07367,test,c44311c2967a48ce9f37d41e5226fa8c,817d161a1f7d3919
Lung_Opacity-4569.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4569.png,Lung_Opacity/masks/Lung_Opacity-4569.png,True,G07368,test,628bf8a0ee6f130760147f9ae76688de,983b1658597baf84
Lung_Opacity-457.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-457.png,Lung_Opacity/masks/Lung_Opacity-457.png,True,G07369,val,03785373fcb79f82180cb452b519d18d,d42ae34b4f4a5c4e
Lung_Opacity-4570.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4570.png,Lung_Opacity/masks/Lung_Opacity-4570.png,True,G07370,val,783ce7d0361510c2cddef0c8557db460,cd2f430f0d293d27
Lung_Opacity-4571.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4571.png,Lung_Opacity/masks/Lung_Opacity-4571.png,True,G07371,train,81f1b15ed64799ac517293c1d5c1cad6,944c075b4d4e5a7e
Lung_Opacity-4572.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4572.png,Lung_Opacity/masks/Lung_Opacity-4572.png,True,G07372,train,cccc7cecf9eaf7f6c899167485a323f4,ca3579d1554b6968
Lung_Opacity-4573.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4573.png,Lung_Opacity/masks/Lung_Opacity-4573.png,True,G07373,test,0c8b47b225f24dec04e2a8700423a5e1,997b565a590a1b2b
Lung_Opacity-4574.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4574.png,Lung_Opacity/masks/Lung_Opacity-4574.png,True,G07374,train,ad95af82f4bd40417aa2dc2563abc9f4,817f06061e1f1f1f
Lung_Opacity-4575.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4575.png,Lung_Opacity/masks/Lung_Opacity-4575.png,True,G07375,train,9145ab410db9c8c9ac710732b7e67250,d02d6fff4b4a4a08
Lung_Opacity-4576.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4576.png,Lung_Opacity/masks/Lung_Opacity-4576.png,True,G07376,train,240d17eae123a5221559920333c22651,956d0ac61e5daf48
Lung_Opacity-4577.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4577.png,Lung_Opacity/masks/Lung_Opacity-4577.png,True,G07377,train,05792103df1107b5a773d7fd34df89d4,c03dff02fe00fd0a
Lung_Opacity-4578.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4578.png,Lung_Opacity/masks/Lung_Opacity-4578.png,True,G07378,train,bd707608f78c2e5120da358681024839,d52ac34b0f2e7233
Lung_Opacity-4579.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4579.png,Lung_Opacity/masks/Lung_Opacity-4579.png,True,G07379,train,81149b379cd124570bda6b31f0efb528,d039ff56964a425c
Lung_Opacity-458.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-458.png,Lung_Opacity/masks/Lung_Opacity-458.png,True,G07380,train,35e973bcde6fc5a1be44ff4f605b5d99,c82ec25acd4f4fc8
Lung_Opacity-4580.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4580.png,Lung_Opacity/masks/Lung_Opacity-4580.png,True,G07381,train,4a3af56dafde10cd0a38ad16fa7944e9,d42b071f1e0d0f4f
Lung_Opacity-4581.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4581.png,Lung_Opacity/masks/Lung_Opacity-4581.png,True,G07382,train,62e52274b52a688b9d5441e324939e5f,d83d471a1f4f0c4e
Lung_Opacity-4582.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4582.png,Lung_Opacity/masks/Lung_Opacity-4582.png,True,G07383,train,a8d27fbe403df468ca8e5d6afed6cbf4,8c2a430b282fef3f
Lung_Opacity-4583.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4583.png,Lung_Opacity/masks/Lung_Opacity-4583.png,True,G07384,train,8ebb8651aa377f90b3e2efccda7208e6,c83d4a0b1d3d3d1d
Lung_Opacity-4584.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4584.png,Lung_Opacity/masks/Lung_Opacity-4584.png,True,G07385,train,aae1b01ba4dd3897d1072516b3b43b4f,81791f9c1c5a5a5b
Lung_Opacity-4585.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4585.png,Lung_Opacity/masks/Lung_Opacity-4585.png,True,G07386,val,c51e4c3e0621ecf899ca91d20500cdd7,987f431f4c4d0e4d
Lung_Opacity-4586.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4586.png,Lung_Opacity/masks/Lung_Opacity-4586.png,True,G07387,val,31e9eb91e27187e74d638a90d9f3d52c,8129b606ae54be5f
Lung_Opacity-4587.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4587.png,Lung_Opacity/masks/Lung_Opacity-4587.png,True,G07388,val,3d841ed950abf1b347e28b1fc63e164d,d56f820a0c5f4b7c
Lung_Opacity-4588.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4588.png,Lung_Opacity/masks/Lung_Opacity-4588.png,True,G07389,test,c8a4a5111b2a5c6d2dd54334059690f6,896a96481f5f185f
Lung_Opacity-4589.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4589.png,Lung_Opacity/masks/Lung_Opacity-4589.png,True,G07390,test,2ce3baa5d762971b4728fef844043d66,dd2ec2431b174b2d
Lung_Opacity-459.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-459.png,Lung_Opacity/masks/Lung_Opacity-459.png,True,G07391,train,61138d53866ab63d3c6e909c679a65f4,947c454f4e4e4e4d
Lung_Opacity-4590.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4590.png,Lung_Opacity/masks/Lung_Opacity-4590.png,True,G07392,train,c2585bca424b6a173520922bdd60eb74,d13bfb0a3b3b1230
Lung_Opacity-4591.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4591.png,Lung_Opacity/masks/Lung_Opacity-4591.png,True,G07393,test,ed4ec51f8d1dc3d414a629090310b144,896d861a493f3a6b
Lung_Opacity-4592.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4592.png,Lung_Opacity/masks/Lung_Opacity-4592.png,True,G07394,train,78fa079a7327800932acdf604358d632,9d403e1f1a497979
Lung_Opacity-4593.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4593.png,Lung_Opacity/masks/Lung_Opacity-4593.png,True,G07395,train,76d66b745e62bd9dd401b8a91e8d5f21,856e065d7c6cf1e0
Lung_Opacity-4594.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4594.png,Lung_Opacity/masks/Lung_Opacity-4594.png,True,G07396,train,e47c2059c97cc8bce6a58d7031d920d4,8578ae3e0a951e57
Lung_Opacity-4595.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4595.png,Lung_Opacity/masks/Lung_Opacity-4595.png,True,G07397,train,892a6f2412577de28b3b1a8e3406563d,887dc24a5f4a3d5a
Lung_Opacity-4596.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4596.png,Lung_Opacity/masks/Lung_Opacity-4596.png,True,G07398,val,053261240d422048e591b9195eba4512,842d160f1f5f6978
Lung_Opacity-4597.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4597.png,Lung_Opacity/masks/Lung_Opacity-4597.png,True,G07399,val,cbb0dfb49526bb375ff7575267ff4f4b,817f07061e1e1f1f
Lung_Opacity-4598.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4598.png,Lung_Opacity/masks/Lung_Opacity-4598.png,True,G07400,train,d4ef58a56956cf61f1edc1b9385efa2c,dc6d430f0d2f0d3c
Lung_Opacity-4599.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4599.png,Lung_Opacity/masks/Lung_Opacity-4599.png,True,G07401,test,8e7e1694e8e35c13d89f797d8fa8316c,913b041e1b5f594f
Lung_Opacity-46.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-46.png,Lung_Opacity/masks/Lung_Opacity-46.png,True,G07402,train,276794aa91e71fb8a0ee5da8fa96fc2a,d4ac2fc21b514f4d
Lung_Opacity-460.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-460.png,Lung_Opacity/masks/Lung_Opacity-460.png,True,G07403,val,23d82cc4adc039e387545503710dd7f9,952807161f5d7e5c
Lung_Opacity-4600.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4600.png,Lung_Opacity/masks/Lung_Opacity-4600.png,True,G07404,test,688a97973d15d9f81830d684716b4520,93781d3b7a39a88c
Lung_Opacity-4601.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4601.png,Lung_Opacity/masks/Lung_Opacity-4601.png,True,G07405,train,4f1b5a366ae84cae2b7940fe3fa7cf1b,d42d4a0f0a0fafad
Lung_Opacity-4602.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4602.png,Lung_Opacity/masks/Lung_Opacity-4602.png,True,G07406,train,0d96ac005fd150dfc7552c0d85cf7779,817f1c107bebb584
Lung_Opacity-4603.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4603.png,Lung_Opacity/masks/Lung_Opacity-4603.png,True,G07407,train,efbb2eeecfc1c5e30055a91e038c4569,d42b47421f5a1f1e
Lung_Opacity-4604.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4604.png,Lung_Opacity/masks/Lung_Opacity-4604.png,True,G07408,train,a3a8926d0a8b02d0383a64b8158d91c9,d42dd2d30f4bc30b
Lung_Opacity-4605.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4605.png,Lung_Opacity/masks/Lung_Opacity-4605.png,True,G07409,train,4cdd29b5243652f80b29c6205fbf1fed,c93e464b1c0f3b2e
Lung_Opacity-4606.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4606.png,Lung_Opacity/masks/Lung_Opacity-4606.png,True,G07410,train,fb03addcd88af3ef053afb51410b30ae,cc3a530e2e2c35b7
Lung_Opacity-4607.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4607.png,Lung_Opacity/masks/Lung_Opacity-4607.png,True,G07411,test,5f8e450ae7cd61e48be17a7af9ab8758,817b16145e5b7879
Lung_Opacity-4608.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4608.png,Lung_Opacity/masks/Lung_Opacity-4608.png,True,G07412,train,566778aab0c2a813a13ca04c94c0318d,dd2f841e1e681e69
Lung_Opacity-4609.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4609.png,Lung_Opacity/masks/Lung_Opacity-4609.png,True,G07413,train,18887d18f93e49e6130ba50dd3aae933,c19534c8037fcb2f
Lung_Opacity-461.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-461.png,Lung_Opacity/masks/Lung_Opacity-461.png,True,G07414,train,8b37fcd86667a9066d5b02d0c6dbc49f,d93ad74a5c495139
Lung_Opacity-4610.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4610.png,Lung_Opacity/masks/Lung_Opacity-4610.png,True,G07415,train,52690a3c1bc17635fd867f6c08fbaace,c03d7b4a1d29393d
Lung_Opacity-4611.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4611.png,Lung_Opacity/masks/Lung_Opacity-4611.png,True,G07416,train,47e77170ee0f0aef03b7be7257012651,d03d47c21b0b4f4f
Lung_Opacity-4612.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4612.png,Lung_Opacity/masks/Lung_Opacity-4612.png,True,G07417,train,eaf711f69706a92a860a572088368cee,817d0e4e5e5b4969
Lung_Opacity-4613.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4613.png,Lung_Opacity/masks/Lung_Opacity-4613.png,True,G07418,train,e8ed92c4c5beaf12d4d8ec126a87115d,992a474f0f1b0d1f
Lung_Opacity-4614.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4614.png,Lung_Opacity/masks/Lung_Opacity-4614.png,True,G07419,train,2252da9c4639d35a09331ccec4b079b3,82609c95371f7673
Lung_Opacity-4615.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4615.png,Lung_Opacity/masks/Lung_Opacity-4615.png,True,G07420,train,9ba226c064ac577763fc8fdfe7629b7e,8330471e783e5e5e
Lung_Opacity-4616.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4616.png,Lung_Opacity/masks/Lung_Opacity-4616.png,True,G07421,train,38ed8ad0d83acb138f6b28f6bb75698c,9f601e0c1b1b1e5f
Lung_Opacity-4617.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4617.png,Lung_Opacity/masks/Lung_Opacity-4617.png,True,G07422,train,e3107790aa5531588bcb7f62ea80ece6,cc2d530b0e291bbf
Lung_Opacity-4618.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4618.png,Lung_Opacity/masks/Lung_Opacity-4618.png,True,G07423,train,7cdbc170a55da6679b6fab1960fb7031,dc2ec20b49c31fe9
Lung_Opacity-4619.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4619.png,Lung_Opacity/masks/Lung_Opacity-4619.png,True,G07424,train,82bb6dd04c2a1277db30bb587cb4ab48,d11116292b373f3e
Lung_Opacity-462.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-462.png,Lung_Opacity/masks/Lung_Opacity-462.png,True,G07425,val,7ec423dd229951455612676363d99b74,dc2fd353187c7818
Lung_Opacity-4620.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4620.png,Lung_Opacity/masks/Lung_Opacity-4620.png,True,G07426,test,e6afe7f41dd598bac85685894c145ef4,81796e04de1ffa09
Lung_Opacity-4621.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4621.png,Lung_Opacity/masks/Lung_Opacity-4621.png,True,G07427,test,0d5a2f20c97989266444dce295181719,d92a845745075f57
Lung_Opacity-4622.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4622.png,Lung_Opacity/masks/Lung_Opacity-4622.png,True,G07428,train,0afe16abe5fc89a7a52d17df8fe60b72,d42c42d75f1d6d0c
Lung_Opacity-4623.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4623.png,Lung_Opacity/masks/Lung_Opacity-4623.png,True,G07429,val,6db364cb49958c6bedb9a7041ced612b,d96d869a098d39cd
Lung_Opacity-4624.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4624.png,Lung_Opacity/masks/Lung_Opacity-4624.png,True,G07430,val,dc9629591d1baaadc948960bf1419509,d42a434b4f4f0d2f
Lung_Opacity-4625.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4625.png,Lung_Opacity/masks/Lung_Opacity-4625.png,True,G07431,test,20f1f4cbfdbee3c4cb6c8116287d2d0e,d56e460f4e2c2ce5
Lung_Opacity-4626.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4626.png,Lung_Opacity/masks/Lung_Opacity-4626.png,True,G07432,train,628f3848ea9743bef8657a1dc913611e,dc2e434b0d0c3ebe
Lung_Opacity-4627.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4627.png,Lung_Opacity/masks/Lung_Opacity-4627.png,True,G07433,train,81332045e6cc03775ac28a97cb909ddf,c43c134b0f6e6d3a
Lung_Opacity-4628.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4628.png,Lung_Opacity/masks/Lung_Opacity-4628.png,True,G07434,train,66d3f78571f83874df5f631012ffe2b9,91782ec61b594d4f
Lung_Opacity-4629.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4629.png,Lung_Opacity/masks/Lung_Opacity-4629.png,True,G07435,train,4f3c7b1481028d31171be466f8460c58,887d07435e7c696a
Lung_Opacity-463.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-463.png,Lung_Opacity/masks/Lung_Opacity-463.png,True,G07436,val,c4618f7e2fa3bf5757b9b91213cd03b9,917b8c941a5b5b2b
Lung_Opacity-4630.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4630.png,Lung_Opacity/masks/Lung_Opacity-4630.png,True,G07437,train,cdf6b07c83052fccfe8ddd25fbfe6167,885f061a49adf8e7
Lung_Opacity-4631.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4631.png,Lung_Opacity/masks/Lung_Opacity-4631.png,True,G07438,train,542dad5ab31624b15f81f9e3390db8a2,d129161b4f1f1c5d
Lung_Opacity-4632.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4632.png,Lung_Opacity/masks/Lung_Opacity-4632.png,True,G07439,train,4218b534b14a327bd85778d67d7a3322,d93a961b392153b9
Lung_Opacity-4633.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4633.png,Lung_Opacity/masks/Lung_Opacity-4633.png,True,G07440,train,65c43ed7bc7efbab2ed7a94364b3fff9,d57f02c30b1e0f0f
Lung_Opacity-4634.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4634.png,Lung_Opacity/masks/Lung_Opacity-4634.png,True,G07441,train,a942d9bc6f72ae5930f2e5d8a036e4ff,8078ff06bc12df1a
Lung_Opacity-4635.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4635.png,Lung_Opacity/masks/Lung_Opacity-4635.png,True,G07442,train,9ee55efdf870dfd9b8de341a669b45b2,807816c31b1d6e7f
Lung_Opacity-4636.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4636.png,Lung_Opacity/masks/Lung_Opacity-4636.png,True,G07443,test,089c0965419aa4b5c605ac290f490c6a,997d46161ae968ba
Lung_Opacity-4637.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4637.png,Lung_Opacity/masks/Lung_Opacity-4637.png,True,G07444,train,a25a0804cd7115ddc4d3cc09720a7fb6,ee145a5bca0acb4b
Lung_Opacity-4638.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4638.png,Lung_Opacity/masks/Lung_Opacity-4638.png,True,G07445,train,7d2549a9a38596c3ec16e353145e16e7,85781c151f1d7d59
Lung_Opacity-4639.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4639.png,Lung_Opacity/masks/Lung_Opacity-4639.png,True,G07446,train,585a77aa0bb0fa9448d31a26d59e850d,817fcc87134e035f
Lung_Opacity-464.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-464.png,Lung_Opacity/masks/Lung_Opacity-464.png,True,G07447,train,667f45921dac59f90e590eca149e39c0,d979c6d61ec45a90
Lung_Opacity-4640.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4640.png,Lung_Opacity/masks/Lung_Opacity-4640.png,True,G07448,val,47dc864859bdfbec4968b86249ada540,c46d0f0a4b6f4e6c
Lung_Opacity-4641.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4641.png,Lung_Opacity/masks/Lung_Opacity-4641.png,True,G07449,val,765d01c336afff6ff9628c7befdcbda8,817bee8e90035b5b
Lung_Opacity-4642.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4642.png,Lung_Opacity/masks/Lung_Opacity-4642.png,True,G07450,test,c18a7c755ca42c27044fa37ceb56f751,dc2e031f0c6c5e4f
Lung_Opacity-4643.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4643.png,Lung_Opacity/masks/Lung_Opacity-4643.png,True,G07451,test,2dabe58c64a51f7f5baa865a6f7ad682,d07a515b182b1f9b
Lung_Opacity-4644.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4644.png,Lung_Opacity/masks/Lung_Opacity-4644.png,True,G07452,train,49b54634594d0f539ee2681494c945a0,d43fdb084d4b0b0f
Lung_Opacity-4645.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4645.png,Lung_Opacity/masks/Lung_Opacity-4645.png,True,G07453,train,24eececb3315329d272b036f157efab0,d42ce6d39b4fe120
Lung_Opacity-4646.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4646.png,Lung_Opacity/masks/Lung_Opacity-4646.png,True,G07454,train,a1c952e12e3f643d8d09f5a03d89fad8,c5bc0f1b0e5c7cc8
Lung_Opacity-4647.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4647.png,Lung_Opacity/masks/Lung_Opacity-4647.png,True,G07455,train,87277a2613872aeb5df9ad1966678334,9179c6191b2d39bc
Lung_Opacity-4648.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4648.png,Lung_Opacity/masks/Lung_Opacity-4648.png,True,G07456,train,ab3ad32393c486e4821cc7e311232e1e,8538108e5f5f7959
Lung_Opacity-4649.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4649.png,Lung_Opacity/masks/Lung_Opacity-4649.png,True,G07457,train,e99039ed71d12f803bdd4d959ffa6658,d470cf9b0b0f2d19
Lung_Opacity-465.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-465.png,Lung_Opacity/masks/Lung_Opacity-465.png,True,G07458,train,4ec3417d7b99c1697a7004d58b007f6a,dc2f410b0f1e3cb6
Lung_Opacity-4650.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4650.png,Lung_Opacity/masks/Lung_Opacity-4650.png,True,G07459,train,ba722e98a19abc1f433d93d9f5bc4ed3,983bc5961e4c5f1a
Lung_Opacity-4651.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4651.png,Lung_Opacity/masks/Lung_Opacity-4651.png,True,G07460,train,e36d5f04c0ad3a1dfe51b03654c11e19,952a82c3cbcbccdc
Lung_Opacity-4652.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4652.png,Lung_Opacity/masks/Lung_Opacity-4652.png,True,G07461,train,aa6ea9036ca5e1aa165b85bf13973af0,8178141a5d7b59ee
Lung_Opacity-4653.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4653.png,Lung_Opacity/masks/Lung_Opacity-4653.png,True,G07462,train,436e28107ec722c8c6a57de3a436e6f1,d17b85869e85ce1a
Lung_Opacity-4654.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4654.png,Lung_Opacity/masks/Lung_Opacity-4654.png,True,G07463,train,4f1f90263f246d5896c3be07a4a942fb,9568061e1a5f5f1e
Lung_Opacity-4655.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4655.png,Lung_Opacity/masks/Lung_Opacity-4655.png,True,G07464,val,4b578db07ac991902859ceac03b7f58b,8138de1efa196959
Lung_Opacity-4656.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4656.png,Lung_Opacity/masks/Lung_Opacity-4656.png,True,G07465,train,af13206bb6cc998756433b683389c74a,d879c71e18535b58
Lung_Opacity-4657.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4657.png,Lung_Opacity/masks/Lung_Opacity-4657.png,True,G07466,train,c64ba54ef79ee6e211979c39e732b059,85681e864a59edee
Lung_Opacity-4658.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4658.png,Lung_Opacity/masks/Lung_Opacity-4658.png,True,G07467,val,e520b6f36d6e007d6fefcc5f18607b8a,dc4d431b0b4f0e3e
Lung_Opacity-4659.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4659.png,Lung_Opacity/masks/Lung_Opacity-4659.png,True,G07468,test,8e675e0a96cae529c2e4fc7c31294434,917f861a3d593919
Lung_Opacity-466.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-466.png,Lung_Opacity/masks/Lung_Opacity-466.png,True,G07469,val,390030db3c35d5056bb668a42020194f,8dee4f4968e1f230
Lung_Opacity-4660.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4660.png,Lung_Opacity/masks/Lung_Opacity-4660.png,True,G07470,test,1603f783defb28560b93d2456551b67d,d42d534b4a4f5a4d
Lung_Opacity-4661.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4661.png,Lung_Opacity/masks/Lung_Opacity-4661.png,True,G07471,test,212e61e99bf2ef0484b213a223c38e28,d42c155b155a2f1f
Lung_Opacity-4662.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4662.png,Lung_Opacity/masks/Lung_Opacity-4662.png,True,G07472,train,cb3fc3392e75d1e1b5feff2d801d5ae4,95686e9660876b9d
Lung_Opacity-4663.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4663.png,Lung_Opacity/masks/Lung_Opacity-4663.png,True,G07473,train,d0eeee2923ffd0974e7eddce598627be,91381f5e5969e1e1
Lung_Opacity-4664.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4664.png,Lung_Opacity/masks/Lung_Opacity-4664.png,True,G07474,val,d95af7aa10b351c2979afca203dece1b,d42dc34b0e0e5f1d
Lung_Opacity-4665.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4665.png,Lung_Opacity/masks/Lung_Opacity-4665.png,True,G07475,train,75c0e3718b8afa510ea4695771d8c9e9,d568161f591d5b49
Lung_Opacity-4666.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4666.png,Lung_Opacity/masks/Lung_Opacity-4666.png,True,G07476,train,018568915cbaf4a697a8e4d43b14d0a3,99655e1a093d69bc
Lung_Opacity-4667.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4667.png,Lung_Opacity/masks/Lung_Opacity-4667.png,True,G07477,test,0223587d5cf6d3534c854fd91815c367,dc0ac74d0e4b4b5b
Lung_Opacity-4668.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4668.png,Lung_Opacity/masks/Lung_Opacity-4668.png,True,G07478,val,cc09bbf21aa42bf2901d8a0055f922a8,9c38c3c71e56c31b
Lung_Opacity-4669.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4669.png,Lung_Opacity/masks/Lung_Opacity-4669.png,True,G07479,test,c46b2a7c92f1c7887fd3d2cb89ee9f14,9179024e5f1b5d1d
Lung_Opacity-467.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-467.png,Lung_Opacity/masks/Lung_Opacity-467.png,True,G07480,train,c10e36f268aa5f8959e03fd9790266ff,c97d14521b1d5b59
Lung_Opacity-4670.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4670.png,Lung_Opacity/masks/Lung_Opacity-4670.png,True,G07481,train,d7656d164d24e7448b51839f233d12ed,903a43438f5f5b1b
Lung_Opacity-4671.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4671.png,Lung_Opacity/masks/Lung_Opacity-4671.png,True,G07482,train,274505073acda8437e7d51ebc6aa886b,817edc5a5b5a7848
Lung_Opacity-4672.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4672.png,Lung_Opacity/masks/Lung_Opacity-4672.png,True,G07483,train,08ca2617dc199e4b19afb5eff5be57ba,cd2c02581f507f7d
Lung_Opacity-4673.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4673.png,Lung_Opacity/masks/Lung_Opacity-4673.png,True,G07484,train,715849ee8cda8e5205bba1aa5af54dbb,c47c2d51c257171f
Lung_Opacity-4674.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4674.png,Lung_Opacity/masks/Lung_Opacity-4674.png,True,G07485,train,528e3b613e748a40241f12341a64c04d,817fae14b8d7421d
Lung_Opacity-4675.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4675.png,Lung_Opacity/masks/Lung_Opacity-4675.png,True,G07486,train,2abd7b232a41c329989d61c8e623154a,cd3c13430f0ced1f
Lung_Opacity-4676.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4676.png,Lung_Opacity/masks/Lung_Opacity-4676.png,True,G07487,train,b8c40b0d10574682b31c95cdfbc7b7ee,dd0ac31f7c903ce4
Lung_Opacity-4677.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4677.png,Lung_Opacity/masks/Lung_Opacity-4677.png,True,G07488,train,2489b0912ada6155d5f9c1c5f1662bd5,f9e78492a3398b89
Lung_Opacity-4678.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4678.png,Lung_Opacity/masks/Lung_Opacity-4678.png,True,G07489,train,e52d42dc7184fbafe2da04c26e919444,d87d0a1f0e2c3cbc
Lung_Opacity-4679.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4679.png,Lung_Opacity/masks/Lung_Opacity-4679.png,True,G07490,val,25ca22165b83785c03665c5dfd532821,846e07d71b5d2a6c
Lung_Opacity-468.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-468.png,Lung_Opacity/masks/Lung_Opacity-468.png,True,G07491,test,91c2a64b6488328787f850527658abb9,dd09171a1d29397d
Lung_Opacity-4680.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4680.png,Lung_Opacity/masks/Lung_Opacity-4680.png,True,G07492,train,ab82d24f551bfb762b53cdc1b5417c9f,d0190a5e1e5f587b
Lung_Opacity-4681.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4681.png,Lung_Opacity/masks/Lung_Opacity-4681.png,True,G07493,train,9b6409951cd0bce36d84a2d4d2075cb4,d02b461939313bbf
Lung_Opacity-4682.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4682.png,Lung_Opacity/masks/Lung_Opacity-4682.png,True,G07494,train,c6dd43c3d155378495fc9100d9c11565,d13f022ef5e59298
Lung_Opacity-4683.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4683.png,Lung_Opacity/masks/Lung_Opacity-4683.png,True,G07495,train,7cd92d2e5193a7d3c7181aaba1dbd454,d42d03435f0e4f4f
Lung_Opacity-4684.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4684.png,Lung_Opacity/masks/Lung_Opacity-4684.png,True,G07496,train,cf8666fe1a0469267c40a59179233b13,9039c79696c39f98
Lung_Opacity-4685.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4685.png,Lung_Opacity/masks/Lung_Opacity-4685.png,True,G07497,train,edc7bb6772011f3cbebe5c19e96773e6,d42d5353524b6b1b
Lung_Opacity-4686.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4686.png,Lung_Opacity/masks/Lung_Opacity-4686.png,True,G07498,train,5cdbe5e9ee995213d6eed4e61d447d59,d929461e1c4f1a3f
Lung_Opacity-4687.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4687.png,Lung_Opacity/masks/Lung_Opacity-4687.png,True,G07499,train,671102553549ef51b7a4c7827e965843,bc1c53d3534e431b
Lung_Opacity-4688.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4688.png,Lung_Opacity/masks/Lung_Opacity-4688.png,True,G07500,train,f32550528e4842551221ff08d20bd416,d83d5bdb0d1c181e
Lung_Opacity-4689.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4689.png,Lung_Opacity/masks/Lung_Opacity-4689.png,True,G07501,val,8d429811fb3385095d63dc82da7172d2,9979845a5b133b39
Lung_Opacity-469.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-469.png,Lung_Opacity/masks/Lung_Opacity-469.png,True,G07502,train,3edd7f8c445daac5e943949892b5f6d2,987b431b4c1b3d17
Lung_Opacity-4690.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4690.png,Lung_Opacity/masks/Lung_Opacity-4690.png,True,G07503,train,2399c613cfe5bad0cfcaa5bc22c7655b,cc3c434b1bed2db0
Lung_Opacity-4691.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4691.png,Lung_Opacity/masks/Lung_Opacity-4691.png,True,G07504,train,3629b5ec537e4889e971a120680724d1,937a0c1a3b3b5a59
Lung_Opacity-4692.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4692.png,Lung_Opacity/masks/Lung_Opacity-4692.png,True,G07505,val,ac0b0f4b04bac4e6296a8568edb7e3fc,cd46827de47ae292
Lung_Opacity-4693.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4693.png,Lung_Opacity/masks/Lung_Opacity-4693.png,True,G07506,train,fdf3da2bcd0ae431e87260c6d51eabb4,896d061e5c5d595b
Lung_Opacity-4694.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4694.png,Lung_Opacity/masks/Lung_Opacity-4694.png,True,G07507,test,185d294755b4fc8fd23f253ef2c39453,d42f6a4547155747
Lung_Opacity-4695.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4695.png,Lung_Opacity/masks/Lung_Opacity-4695.png,True,G07508,test,2a7038b2de53ba2d169a30f1d9bdae28,d12b0b1a3bf8b391
Lung_Opacity-4696.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4696.png,Lung_Opacity/masks/Lung_Opacity-4696.png,True,G07509,train,532285019bdbc8596d4a75b289f0badd,d17f10101f3eb4b5
Lung_Opacity-4697.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4697.png,Lung_Opacity/masks/Lung_Opacity-4697.png,True,G07510,test,1f312a295f8936a30aa8de80c2cd51f5,991a1238b9bde4f1
Lung_Opacity-4698.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4698.png,Lung_Opacity/masks/Lung_Opacity-4698.png,True,G07511,train,1a15bcb6d6f6ecdff30874bdb6bf7aa8,ed2d1703491d3c4f
Lung_Opacity-4699.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4699.png,Lung_Opacity/masks/Lung_Opacity-4699.png,True,G07512,val,0dc1054802da597f63f26f009446db92,ce554b0b0a2f2f2d
Lung_Opacity-47.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-47.png,Lung_Opacity/masks/Lung_Opacity-47.png,True,G07513,train,aef627d5a3925525eb08d218c20d4325,897a06061a5f4f7b
Lung_Opacity-470.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-470.png,Lung_Opacity/masks/Lung_Opacity-470.png,True,G07514,train,9e380bb71da8c551a310f4ccbf600228,c93b16161e1e1b1f
Lung_Opacity-4700.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4700.png,Lung_Opacity/masks/Lung_Opacity-4700.png,True,G07515,train,bcd277e7d2412c4b46d5c8f44295cfa4,9169141c5b5b5b5b
Lung_Opacity-4701.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4701.png,Lung_Opacity/masks/Lung_Opacity-4701.png,True,G07516,train,4561b9c0213fc807e0da035808ccc9d3,cd3e5a090c0d97e7
Lung_Opacity-4702.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4702.png,Lung_Opacity/masks/Lung_Opacity-4702.png,True,G07517,train,41079d3f753be763e02177fbe27bde24,9781e129d12fda47
Lung_Opacity-4703.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4703.png,Lung_Opacity/masks/Lung_Opacity-4703.png,True,G07518,train,a942dac9106e37ce8844d485edde8b6d,953f47461e581a5e
Lung_Opacity-4704.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4704.png,Lung_Opacity/masks/Lung_Opacity-4704.png,True,G07519,train,8c01cf9381563d2cc074b8f6e8381de8,807f875e19584b7a
Lung_Opacity-4705.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4705.png,Lung_Opacity/masks/Lung_Opacity-4705.png,True,G07520,train,6d9068c9c32774c2eb1420896ba9b860,917a87165a5b5978
Lung_Opacity-4706.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4706.png,Lung_Opacity/masks/Lung_Opacity-4706.png,True,G07521,train,ce1c5ced647f98b6cc0a20b12d12411c,c52e4778ece4a0a7
Lung_Opacity-4707.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4707.png,Lung_Opacity/masks/Lung_Opacity-4707.png,True,G07522,train,1f8aab1fc6f16a5f299b610468bc88d9,cc3c430b0fac2fbc
Lung_Opacity-4708.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4708.png,Lung_Opacity/masks/Lung_Opacity-4708.png,True,G07523,train,28972344efe9ec44ba81959a79b03bc3,dc2d52d34b4b4b49
Lung_Opacity-4709.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4709.png,Lung_Opacity/masks/Lung_Opacity-4709.png,True,G07524,val,68b8b38def52d60ffdbfcc05e4f6ab40,dc28c34b5c557636
Lung_Opacity-471.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-471.png,Lung_Opacity/masks/Lung_Opacity-471.png,True,G07525,test,166ad6aa4855897f92bf108e01bbd3c9,c0bb50185e1d5e5f
Lung_Opacity-4710.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4710.png,Lung_Opacity/masks/Lung_Opacity-4710.png,True,G07526,val,ff8b62eb5470efeabfa30720829e5c33,81789f5b585c7869
Lung_Opacity-4711.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4711.png,Lung_Opacity/masks/Lung_Opacity-4711.png,True,G07527,train,720f1288d2f5e552e760b1049f8a1e18,807b07d61e78697c
Lung_Opacity-4712.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4712.png,Lung_Opacity/masks/Lung_Opacity-4712.png,True,G07528,train,afe6be42af8b905e10ecb19c0590bf8a,812e861f595d5979
Lung_Opacity-4713.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4713.png,Lung_Opacity/masks/Lung_Opacity-4713.png,True,G07529,val,2366dde57f960c2f3fd013026deef431,8869d6c2dbdda8c1
Lung_Opacity-4714.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4714.png,Lung_Opacity/masks/Lung_Opacity-4714.png,True,G07530,test,bf45634a1675d7d0bb7347494460f635,d63ef0d0990dc6d2
Lung_Opacity-4715.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4715.png,Lung_Opacity/masks/Lung_Opacity-4715.png,True,G07531,test,641e1a3ca2065fd2320586c5219fc988,802b465f1f0d6f1e
Lung_Opacity-4716.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4716.png,Lung_Opacity/masks/Lung_Opacity-4716.png,True,G07532,train,55a79531225b771c69bb70cfc07edfa1,913fd61658185d5b
Lung_Opacity-4717.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4717.png,Lung_Opacity/masks/Lung_Opacity-4717.png,True,G07533,train,dbbfb7d52086d52eb246fe27d6f76254,9c666391668c6f9c
Lung_Opacity-4718.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4718.png,Lung_Opacity/masks/Lung_Opacity-4718.png,True,G07534,val,a85367f8a975a23e1beedd0a284b9acf,987f921ec51d161e
Lung_Opacity-4719.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4719.png,Lung_Opacity/masks/Lung_Opacity-4719.png,True,G07535,train,8aca4e27b2fcf540fccdc02bb67d1af0,ca3d6101dd49d2db
Lung_Opacity-472.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-472.png,Lung_Opacity/masks/Lung_Opacity-472.png,True,G07536,train,8b9f0224ce7c7696afd30484f3bc43ea,d13b45cf59184e0f
Lung_Opacity-4720.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4720.png,Lung_Opacity/masks/Lung_Opacity-4720.png,True,G07537,train,71b4c357828d245e09f79bda560c90f0,d51e69c90d1e9c36
Lung_Opacity-4721.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4721.png,Lung_Opacity/masks/Lung_Opacity-4721.png,True,G07538,train,4be5a67fc2f15e94f04ca819fbfb1d72,c03d56911d5b595b
Lung_Opacity-4722.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4722.png,Lung_Opacity/masks/Lung_Opacity-4722.png,True,G07539,train,030864c0dd72d7fe453c05e0df9e1714,846e834b4d474f4f
Lung_Opacity-4723.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4723.png,Lung_Opacity/masks/Lung_Opacity-4723.png,True,G07540,train,22b77396f371515dc5cd26a08273a82d,9c66831b457c497e
Lung_Opacity-4724.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4724.png,Lung_Opacity/masks/Lung_Opacity-4724.png,True,G07541,train,54f2c8d8b000520cf37f34f178928514,c93d17121c5d5b59
Lung_Opacity-4725.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4725.png,Lung_Opacity/masks/Lung_Opacity-4725.png,True,G07542,test,2ee4665ec2f3f56830ae87b26aa7254e,957f42570a2b2c3d
Lung_Opacity-4726.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4726.png,Lung_Opacity/masks/Lung_Opacity-4726.png,True,G07543,test,1b1b310fae47f05df5e7387207dfc02d,c42f1a595c2c6e6d
Lung_Opacity-4727.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4727.png,Lung_Opacity/masks/Lung_Opacity-4727.png,True,G07544,train,7a29247bd21c95777f34ff87bd6c0df2,8c0c434f4d4d7e7a
Lung_Opacity-4728.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4728.png,Lung_Opacity/masks/Lung_Opacity-4728.png,True,G07545,train,b7d309ad9c2e9284d7fcf0231cf02dbb,dc2e430f4c3c2c6f
Lung_Opacity-4729.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4729.png,Lung_Opacity/masks/Lung_Opacity-4729.png,True,G07546,train,cdbb4ffa6b531086b9e323eb299d8d2c,816a80195b5ff95d
Lung_Opacity-473.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-473.png,Lung_Opacity/masks/Lung_Opacity-473.png,True,G07547,train,638e75a199e9ab2785e5974820c7f1fe,c12f345a494b7b3c
Lung_Opacity-4730.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4730.png,Lung_Opacity/masks/Lung_Opacity-4730.png,True,G07548,train,1fe9debb32e390d5f1dadd678e092bd3,c93b168a194de9b3
Lung_Opacity-4731.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4731.png,Lung_Opacity/masks/Lung_Opacity-4731.png,True,G07549,val,0cb8002a97ba5074bccd7bfc221c959c,872e9e70385538d7
Lung_Opacity-4732.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4732.png,Lung_Opacity/masks/Lung_Opacity-4732.png,True,G07550,val,03200c1ccf50ab11051ccba2a2e148a0,d42f41494f0f0f4f
Lung_Opacity-4733.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4733.png,Lung_Opacity/masks/Lung_Opacity-4733.png,True,G07551,train,ad2bc671840fdccd76b2e8d29276dbb3,803f4e4b6c2cbce3
Lung_Opacity-4734.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4734.png,Lung_Opacity/masks/Lung_Opacity-4734.png,True,G07552,test,07de20dcd219cc4f9495e77e7b9eb9ff,cc3ddb4b0e0d2e0d
Lung_Opacity-4735.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4735.png,Lung_Opacity/masks/Lung_Opacity-4735.png,True,G07553,train,c0d4ba6ad7082938bc1547ac85b4886b,95788e1e1d1c3a1f
Lung_Opacity-4736.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4736.png,Lung_Opacity/masks/Lung_Opacity-4736.png,True,G07554,train,8c9d291f8f14a627cd4ea9ad6ed14be4,91695f5a197b1998
Lung_Opacity-4737.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4737.png,Lung_Opacity/masks/Lung_Opacity-4737.png,True,G07555,train,b0743241d19c6f4d8b7d5f2624502fef,de2de105450d2f67
Lung_Opacity-4738.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4738.png,Lung_Opacity/masks/Lung_Opacity-4738.png,True,G07556,train,a3808ada1ec0da2c656f494b8728b348,d86c470a4c6c1f6f
Lung_Opacity-4739.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4739.png,Lung_Opacity/masks/Lung_Opacity-4739.png,True,G07557,train,bc070642451bc39e1c727fb841972ed5,997a1218adb7b4a6
Lung_Opacity-474.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-474.png,Lung_Opacity/masks/Lung_Opacity-474.png,True,G07558,train,6ebe0229fc31d86f6ea149ea80382ff2,817b44025b7b7979
Lung_Opacity-4740.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4740.png,Lung_Opacity/masks/Lung_Opacity-4740.png,True,G07559,val,b2c4fd644b073a2cd6b2576e618c3961,d17b521e18393f34
Lung_Opacity-4741.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4741.png,Lung_Opacity/masks/Lung_Opacity-4741.png,True,G07560,train,ad846abf61c489341fda5f262b3c00d1,846e454b2e2eaabb
Lung_Opacity-4742.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4742.png,Lung_Opacity/masks/Lung_Opacity-4742.png,True,G07561,test,72b237fc2345126d0fdb54e74949b01c,890d065a596d79fc
Lung_Opacity-4743.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4743.png,Lung_Opacity/masks/Lung_Opacity-4743.png,True,G07562,test,b8d5af8779a4b31ca97f41c46ebc206b,9179c31f1fc60d19
Lung_Opacity-4744.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4744.png,Lung_Opacity/masks/Lung_Opacity-4744.png,True,G07563,train,4585c1e7054e4b323cbd26e5efe1a241,913b4e061e5e5a5b
Lung_Opacity-4745.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4745.png,Lung_Opacity/masks/Lung_Opacity-4745.png,True,G07564,train,defb059dc490b3e8aa4625f2808fb9cb,c42e434b4f4b4f69
Lung_Opacity-4746.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4746.png,Lung_Opacity/masks/Lung_Opacity-4746.png,True,G07565,train,1ce56fb2c34e71b02198e75db1b26d0b,803d641a5a5b7b6b
Lung_Opacity-4747.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4747.png,Lung_Opacity/masks/Lung_Opacity-4747.png,True,G07566,train,87235221b126d8395474f2c2931439e1,8464171a1f3e1c7f
Lung_Opacity-4748.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4748.png,Lung_Opacity/masks/Lung_Opacity-4748.png,True,G07567,val,e52030811086be61eff83cd0949bdd8c,917b861a155e1a5f
Lung_Opacity-4749.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4749.png,Lung_Opacity/masks/Lung_Opacity-4749.png,True,G07568,train,ee6ce611bbaf50d4c81aaf8acd344fbf,c13e181d4dbd96c5
Lung_Opacity-475.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-475.png,Lung_Opacity/masks/Lung_Opacity-475.png,True,G07569,train,f926f4e974945bf030a2dbf41f112f98,dd2dc3070f0d1e0f
Lung_Opacity-4750.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4750.png,Lung_Opacity/masks/Lung_Opacity-4750.png,True,G07570,val,6b725fc8679a32970ea57ea0da35b79f,c5e47772626b2370
Lung_Opacity-4751.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4751.png,Lung_Opacity/masks/Lung_Opacity-4751.png,True,G07571,test,36b3a8a87c4aa039b7f30eb51fabff4a,8566c6586d6e6666
Lung_Opacity-4752.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4752.png,Lung_Opacity/masks/Lung_Opacity-4752.png,True,G07572,train,4d39d8f4c6751e617dcac39f25506fe4,83081e1e5f7a7a78
Lung_Opacity-4753.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4753.png,Lung_Opacity/masks/Lung_Opacity-4753.png,True,G07573,train,ea892853baa0cfd15aff79402e360f33,807f047a6767e4e4
Lung_Opacity-4754.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4754.png,Lung_Opacity/masks/Lung_Opacity-4754.png,True,G07574,train,20880f36f3fa96ffc3fcfd711f914fcf,d56a53434f0d2e4e
Lung_Opacity-4755.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4755.png,Lung_Opacity/masks/Lung_Opacity-4755.png,True,G07575,train,54d75a99fe65b44938b2fb7857452b36,dd6a454b424b1f2b
Lung_Opacity-4756.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4756.png,Lung_Opacity/masks/Lung_Opacity-4756.png,True,G07576,train,d8549b55b30d68a0944fdd1b318ec516,917dc61668697953
Lung_Opacity-4757.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4757.png,Lung_Opacity/masks/Lung_Opacity-4757.png,True,G07577,train,04b13e43f911753a9c080a7c5ac5bbd6,dd7f861e49054a4d
Lung_Opacity-4758.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4758.png,Lung_Opacity/masks/Lung_Opacity-4758.png,True,G07578,train,cb36413bc87e5eaff0c92a5c76ef650e,c13b5206595b6b6b
Lung_Opacity-4759.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4759.png,Lung_Opacity/masks/Lung_Opacity-4759.png,True,G07579,train,d36a58751a0ce6dfc69a01526a8193f9,d16a0596171b1f1f
Lung_Opacity-476.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-476.png,Lung_Opacity/masks/Lung_Opacity-476.png,True,G07580,train,282b6ab1b9f9c881e3f4c1414f762985,996e07120f1d1f4f
Lung_Opacity-4760.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4760.png,Lung_Opacity/masks/Lung_Opacity-4760.png,True,G07581,train,3bc89d0459a9490f65dbf8f472900267,987bd71a595b1909
Lung_Opacity-4761.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4761.png,Lung_Opacity/masks/Lung_Opacity-4761.png,True,G07582,train,567cbc9c947863aa515c0ccf1519012e,dd3fc0165b411eb8
Lung_Opacity-4762.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4762.png,Lung_Opacity/masks/Lung_Opacity-4762.png,True,G07583,val,9147a717de2ab998bbb90a217dde57f2,95e0ac05171f1f1f
Lung_Opacity-4763.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4763.png,Lung_Opacity/masks/Lung_Opacity-4763.png,True,G07584,test,153ea2f46e4d1aabf8a57802432453c4,81a8961f1e9e1e97
Lung_Opacity-4764.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4764.png,Lung_Opacity/masks/Lung_Opacity-4764.png,True,G07585,train,fc00068864c6230d68dfeff6cf5589d5,997b861b1989baa5
Lung_Opacity-4765.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4765.png,Lung_Opacity/masks/Lung_Opacity-4765.png,True,G07586,train,d22c36a28bd16f47ebcdc22614144fac,996dc7164d471e4a
Lung_Opacity-4766.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4766.png,Lung_Opacity/masks/Lung_Opacity-4766.png,True,G07587,train,2bf77f0296e482a109a2b8620daa68fa,d0354b574a0b4f4f
Lung_Opacity-4767.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4767.png,Lung_Opacity/masks/Lung_Opacity-4767.png,True,G07588,val,492e2e731a4c80922b1c86ff684c0f6c,903c074acf4f7c6c
Lung_Opacity-4768.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4768.png,Lung_Opacity/masks/Lung_Opacity-4768.png,True,G07589,val,8392326efca8ab6b69a6588587036c12,cc35834b0f4b0f4f
Lung_Opacity-4769.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4769.png,Lung_Opacity/masks/Lung_Opacity-4769.png,True,G07590,test,282ec4ba76362596a1ec4b7ab938bdc3,9139c6561e5d4f1a
Lung_Opacity-477.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-477.png,Lung_Opacity/masks/Lung_Opacity-477.png,True,G07591,test,a88141d8a6de74b456f50386b6f6b571,cc3bd34f434b420f
Lung_Opacity-4770.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4770.png,Lung_Opacity/masks/Lung_Opacity-4770.png,True,G07592,test,e3e7b15568c4d2c171dd3c909b748a20,91780e1a4f597979
Lung_Opacity-4771.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4771.png,Lung_Opacity/masks/Lung_Opacity-4771.png,True,G07593,train,148765a44a90910a56a85a13802ed6d5,d8394f8d9cf8e0e0
Lung_Opacity-4772.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4772.png,Lung_Opacity/masks/Lung_Opacity-4772.png,True,G07594,train,d339dea80eda6d925013450c60d22274,c17b5604581fde4e
Lung_Opacity-4773.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4773.png,Lung_Opacity/masks/Lung_Opacity-4773.png,True,G07595,train,ff7ed566cf020a4bfbd999f100a2b1a5,991b061e1a7a7af8
Lung_Opacity-4774.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4774.png,Lung_Opacity/masks/Lung_Opacity-4774.png,True,G07596,train,53881aec23cd0037d178f584675227a7,9152571e3e39a5b4
Lung_Opacity-4775.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4775.png,Lung_Opacity/masks/Lung_Opacity-4775.png,True,G07597,test,c37673b44ad03cabc89666e049618de2,d42a9f1f0c79b838
Lung_Opacity-4776.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4776.png,Lung_Opacity/masks/Lung_Opacity-4776.png,True,G07598,train,a334bc495ada3bfe7e0f2293f86ea0bb,dd2acb0d8a83b3b2
Lung_Opacity-4777.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4777.png,Lung_Opacity/masks/Lung_Opacity-4777.png,True,G07599,train,4068aa8907c0a0b7e7d30173f7804f6b,d42f4a574a0f1b0f
Lung_Opacity-4778.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4778.png,Lung_Opacity/masks/Lung_Opacity-4778.png,True,G07600,train,1097bbe6b52b061d27965df23784d6a6,897987121b595b9e
Lung_Opacity-4779.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4779.png,Lung_Opacity/masks/Lung_Opacity-4779.png,True,G07601,train,9c032b96a95e2e8e98f3ecf5075a0be6,8178861b7b797949
Lung_Opacity-478.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-478.png,Lung_Opacity/masks/Lung_Opacity-478.png,True,G07602,test,f621a2e6727ced0ce6ab061fba041ea3,d03f430e195f383b
Lung_Opacity-4780.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4780.png,Lung_Opacity/masks/Lung_Opacity-4780.png,True,G07603,train,0fe863bc816dbaaeb3776ba3ec6bbeb4,d17b561a19161f1e
Lung_Opacity-4781.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4781.png,Lung_Opacity/masks/Lung_Opacity-4781.png,True,G07604,train,752a6616ac1bda8c87d36b5ccabba109,dc3dcb0f2c2e0d29
Lung_Opacity-4782.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4782.png,Lung_Opacity/masks/Lung_Opacity-4782.png,True,G07605,train,adb28eb2d234b8ec65b86daffe5f15b3,8949dc1f59687959
Lung_Opacity-4783.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4783.png,Lung_Opacity/masks/Lung_Opacity-4783.png,True,G07606,train,0be875c3629a62ccc1c20174afe70db1,f839c5c7161e4353
Lung_Opacity-4784.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4784.png,Lung_Opacity/masks/Lung_Opacity-4784.png,True,G07607,train,65e02accef0b082184f61cafc6c10da9,807f571e5839392b
Lung_Opacity-4785.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4785.png,Lung_Opacity/masks/Lung_Opacity-4785.png,True,G07608,train,e44f811b91fde2228afe335a7a5bef67,8019579a7bc97b68
Lung_Opacity-4786.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4786.png,Lung_Opacity/masks/Lung_Opacity-4786.png,True,G07609,val,7f499e23a3c317dd8e2fee6da1fd4baa,cd2c161f5e1a5a69
Lung_Opacity-4787.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4787.png,Lung_Opacity/masks/Lung_Opacity-4787.png,True,G07610,val,444f15ae4936d5a3d075722386da1d4c,912bfa4419195b5f
Lung_Opacity-4788.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4788.png,Lung_Opacity/masks/Lung_Opacity-4788.png,True,G07611,val,31aec8dbebe863c589e89f98f6025763,913f061c1d4d3d5d
Lung_Opacity-4789.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4789.png,Lung_Opacity/masks/Lung_Opacity-4789.png,True,G07612,train,c1c13d4c2cc392fbce8dd0dc379ab7ac,89795e585b596949
Lung_Opacity-479.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-479.png,Lung_Opacity/masks/Lung_Opacity-479.png,True,G07613,test,322ef6f5431a8ecf14551d139b3d0685,9d6dc55a496c18e5
Lung_Opacity-4790.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4790.png,Lung_Opacity/masks/Lung_Opacity-4790.png,True,G07614,train,4728e8d586c99f5f53d01a499c434b04,952cc7979acac91a
Lung_Opacity-4791.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4791.png,Lung_Opacity/masks/Lung_Opacity-4791.png,True,G07615,train,da979172aef331ba9b4e4b195d044248,9959861ac9bdb8e1
Lung_Opacity-4792.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4792.png,Lung_Opacity/masks/Lung_Opacity-4792.png,True,G07616,train,d37ec8abe708f2cf757ebdcbc5a772aa,dc5a430e0e2dbde4
Lung_Opacity-4793.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4793.png,Lung_Opacity/masks/Lung_Opacity-4793.png,True,G07617,train,a222baf18cea22145128cd78225a3640,997947971e430f38
Lung_Opacity-4794.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4794.png,Lung_Opacity/masks/Lung_Opacity-4794.png,True,G07618,train,82fcafb988179369e28c7ef64d18d29d,8c44d74f4d494b4f
Lung_Opacity-4795.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4795.png,Lung_Opacity/masks/Lung_Opacity-4795.png,True,G07619,train,5505df7099416e8ad7e2b46289d7d1a4,817f0e5a79c984e9
Lung_Opacity-4796.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4796.png,Lung_Opacity/masks/Lung_Opacity-4796.png,True,G07620,train,48ca68695d36857124b4be03b1f82b92,886fc7141e475b4b
Lung_Opacity-4797.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4797.png,Lung_Opacity/masks/Lung_Opacity-4797.png,True,G07621,train,bbe6279b35894c90a5e457fc9184143a,d42d51490c6e3b7b
Lung_Opacity-4798.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4798.png,Lung_Opacity/masks/Lung_Opacity-4798.png,True,G07622,train,2fabd78b80c7204e169292eccff8b7dc,817a07181f4f2f4f
Lung_Opacity-4799.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4799.png,Lung_Opacity/masks/Lung_Opacity-4799.png,True,G07623,train,e8e87a6d0efda5bc17e3afd76b2503ba,ca5d4b4b0e0f2e2d
Lung_Opacity-48.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-48.png,Lung_Opacity/masks/Lung_Opacity-48.png,True,G07624,train,a0eba6dab53e6c73565d849511ac02a8,8370171f3b3a3ab8
Lung_Opacity-480.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-480.png,Lung_Opacity/masks/Lung_Opacity-480.png,True,G07625,train,f3ac60651be9320d6c87c661135a61b3,817b06165f585b5b
Lung_Opacity-4800.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4800.png,Lung_Opacity/masks/Lung_Opacity-4800.png,True,G07626,val,bf3cdaf28e4a6d5a570dc8d2bade38d7,95481e5a5b6b7ac2
Lung_Opacity-4801.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4801.png,Lung_Opacity/masks/Lung_Opacity-4801.png,True,G07627,train,9433b68b3a9a4199992237ce29f6009c,956c87121b0f3d3e
Lung_Opacity-4802.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4802.png,Lung_Opacity/masks/Lung_Opacity-4802.png,True,G07628,val,7e6e5374d4ddf66f2e15f2fc8d01dabb,c17b95069c1d5a9e
Lung_Opacity-4803.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4803.png,Lung_Opacity/masks/Lung_Opacity-4803.png,True,G07629,val,6b3cc93d4b6a22c9863b9ad62600db81,d43e874b2c6e3934
Lung_Opacity-4804.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4804.png,Lung_Opacity/masks/Lung_Opacity-4804.png,True,G07630,test,bb833ff5e9ed608c113d3895ec1cfa4f,d03b831e1e3a3d65
Lung_Opacity-4805.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4805.png,Lung_Opacity/masks/Lung_Opacity-4805.png,True,G07631,test,a3c9ce273e404b2b8f69c9b81b82cf60,807fd74e4a595d09
Lung_Opacity-4806.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4806.png,Lung_Opacity/masks/Lung_Opacity-4806.png,True,G07632,test,a46b3535a65979652a2d3b4b46bfaf93,817a0d041f5d4f5f
Lung_Opacity-4807.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4807.png,Lung_Opacity/masks/Lung_Opacity-4807.png,True,G07633,train,d054949ece27a895afd22c93d1e94a0e,cd0a161a5d78eaea
Lung_Opacity-4808.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4808.png,Lung_Opacity/masks/Lung_Opacity-4808.png,True,G07634,train,ee61acb09ef6488e531719e4f31bc593,816ac61279797d1d
Lung_Opacity-4809.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4809.png,Lung_Opacity/masks/Lung_Opacity-4809.png,True,G07635,train,f8ffcc2470db01e5f286a5cddb6cf1f4,99798610994efe59
Lung_Opacity-481.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-481.png,Lung_Opacity/masks/Lung_Opacity-481.png,True,G07636,train,65610a8499d549d472a1eae4a4f092e9,997b06161d5f1a5a
Lung_Opacity-4810.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4810.png,Lung_Opacity/masks/Lung_Opacity-4810.png,True,G07637,train,fbc4545134dcfcea9c454f7f9482e8d5,992cc6184f5f78e2
Lung_Opacity-4811.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4811.png,Lung_Opacity/masks/Lung_Opacity-4811.png,True,G07638,train,839cd2f8dbfcce78002dc375d001bbec,807bd4121a1f5f3b
Lung_Opacity-4812.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4812.png,Lung_Opacity/masks/Lung_Opacity-4812.png,True,G07639,train,49fd4286470d1fb3667ea398fb445b6e,8978061a1f1f1b5f
Lung_Opacity-4813.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4813.png,Lung_Opacity/masks/Lung_Opacity-4813.png,True,G07640,train,1e46b40610e6e43053191c9efbf681fc,9d2a835f084f4c6f
Lung_Opacity-4814.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4814.png,Lung_Opacity/masks/Lung_Opacity-4814.png,True,G07641,train,d3ea5e224c133adbc596fc13bbdb20d4,cc3d130a2e3d6ccb
Lung_Opacity-4815.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4815.png,Lung_Opacity/masks/Lung_Opacity-4815.png,True,G07642,train,9827262dcda15a2a33a4f8fc58c0b62f,9179060e5e597979
Lung_Opacity-4816.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4816.png,Lung_Opacity/masks/Lung_Opacity-4816.png,True,G07643,train,9dc425c07a533c9f8aec3efb34d8ae3a,dc2de14b0f6a2d0e
Lung_Opacity-4817.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4817.png,Lung_Opacity/masks/Lung_Opacity-4817.png,True,G07644,train,9222a5dddc5b0872702ce56f2f8c3550,882ac75e68efe0e2
Lung_Opacity-4818.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4818.png,Lung_Opacity/masks/Lung_Opacity-4818.png,True,G07645,val,2cc4d9399317a2d2cb7f1167b4db63a2,887b47821b09ed5f
Lung_Opacity-4819.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4819.png,Lung_Opacity/masks/Lung_Opacity-4819.png,True,G07646,train,7041aecf184b1f1caeb42ec95672b7c7,cc2f071b4d4c7c69
Lung_Opacity-482.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-482.png,Lung_Opacity/masks/Lung_Opacity-482.png,True,G07647,train,8a048be45362f58e9e974eada1408402,dc3de1531b44193e
Lung_Opacity-4820.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4820.png,Lung_Opacity/masks/Lung_Opacity-4820.png,True,G07648,val,b15d759522285e80fd22fd44b956a4c2,d52a414f0d0e2f3f
Lung_Opacity-4821.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4821.png,Lung_Opacity/masks/Lung_Opacity-4821.png,True,G07649,val,b36fc3d657de0b20aa6816328c872534,c5284703170f5f5f
Lung_Opacity-4822.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4822.png,Lung_Opacity/masks/Lung_Opacity-4822.png,True,G07650,test,1e73236b1db04d4b5936765dad97cb75,957c060a1e5e5a7e
Lung_Opacity-4823.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4823.png,Lung_Opacity/masks/Lung_Opacity-4823.png,True,G07651,test,2c7e6fa5aec454b2756fcd8ad0608436,907b47461a0b5f1f
Lung_Opacity-4824.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4824.png,Lung_Opacity/masks/Lung_Opacity-4824.png,True,G07652,test,4cb722a206745355e0417c6ffaceceb6,d03dd3d30a0cdb39
Lung_Opacity-4825.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4825.png,Lung_Opacity/masks/Lung_Opacity-4825.png,True,G07653,train,fc1424f97582f88d5af98f68ba6d430d,de25c90f0e2f0f0d
Lung_Opacity-4826.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4826.png,Lung_Opacity/masks/Lung_Opacity-4826.png,True,G07654,train,ee816c4ba9204152ce808983264b242d,cc3cd3431e0b2bbc
Lung_Opacity-4827.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4827.png,Lung_Opacity/masks/Lung_Opacity-4827.png,True,G07655,train,d26fb6401d202c965b732af9e2e7b238,d52ea2d253559e0d
Lung_Opacity-4828.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4828.png,Lung_Opacity/masks/Lung_Opacity-4828.png,True,G07656,train,e61d04c8d22f50e6066ccec63e818aab,d129861e3f4a395b
Lung_Opacity-4829.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4829.png,Lung_Opacity/masks/Lung_Opacity-4829.png,True,G07657,train,8bb2a146bb542467746283fb1068f8f8,81693ed65a591d59
Lung_Opacity-483.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-483.png,Lung_Opacity/masks/Lung_Opacity-483.png,True,G07658,train,53ac6add6b6c5a3b2f0e109dd9f6f5f0,985f434f1c1a4f4b
Lung_Opacity-4830.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4830.png,Lung_Opacity/masks/Lung_Opacity-4830.png,True,G07659,train,56b8d2654d1c3c1525132c8efaa8a20a,c02d47575e4e6aaa
Lung_Opacity-4831.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4831.png,Lung_Opacity/masks/Lung_Opacity-4831.png,True,G07660,train,8b89c400020f21f5de7650d522ffdd0f,d52df7120a4b4e1e
Lung_Opacity-4832.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4832.png,Lung_Opacity/masks/Lung_Opacity-4832.png,True,G07661,train,559641326e9f20f590f33069e98af799,a03b07e6967a096f
Lung_Opacity-4833.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4833.png,Lung_Opacity/masks/Lung_Opacity-4833.png,True,G07662,val,addaf5cf9990eb999c865d050a503fdf,dc2d42534f405f4f
Lung_Opacity-4834.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4834.png,Lung_Opacity/masks/Lung_Opacity-4834.png,True,G07663,train,9c2ada28ce2cf3dd7c238283f3fc0c6a,9d2e471b0c4f5d4c
Lung_Opacity-4835.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4835.png,Lung_Opacity/masks/Lung_Opacity-4835.png,True,G07664,val,39ec350cbc4ac0436ed5f3b70f0cf30e,d93996061915575f
Lung_Opacity-4836.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4836.png,Lung_Opacity/masks/Lung_Opacity-4836.png,True,G07665,test,e509bbf65144a6753ab2ae89a0731ed2,d765400be8276776
Lung_Opacity-4837.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4837.png,Lung_Opacity/masks/Lung_Opacity-4837.png,True,G07666,train,a39c68c5609344933a4226af7999e7cc,990d061e5c5b5d7a
Lung_Opacity-4838.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4838.png,Lung_Opacity/masks/Lung_Opacity-4838.png,True,G07667,val,ca99dc4dbbb1d5c0caae627420f64e55,cd0f4b2ce6e8e4b0
Lung_Opacity-4839.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4839.png,Lung_Opacity/masks/Lung_Opacity-4839.png,True,G07668,test,c499e23814028a87c208287148b72450,dc0d4b0b0a2d3ebd
Lung_Opacity-484.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-484.png,Lung_Opacity/masks/Lung_Opacity-484.png,True,G07669,train,8d00268d32f36aa33a3fc68f2b03faea,942e831f0f4f3c4e
Lung_Opacity-4840.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4840.png,Lung_Opacity/masks/Lung_Opacity-4840.png,True,G07670,test,886958514556ffd187563e7b4597b60e,d52d025f181f0f4f
Lung_Opacity-4841.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4841.png,Lung_Opacity/masks/Lung_Opacity-4841.png,True,G07671,train,bb30040e82b30c120b6c32951c78d9b4,817f066a1f55ea15
Lung_Opacity-4842.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4842.png,Lung_Opacity/masks/Lung_Opacity-4842.png,True,G07672,train,be61be99e9c1fed5331b75939cc5ee76,927f941053135f1f
Lung_Opacity-4843.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4843.png,Lung_Opacity/masks/Lung_Opacity-4843.png,True,G07673,train,d55be8cddb19fee91b33d0135fd35e79,913bc218590f7d5e
Lung_Opacity-4844.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4844.png,Lung_Opacity/masks/Lung_Opacity-4844.png,True,G07674,train,8da62c40e195c83975c2c8d05cc88289,8168961b5f7b5a58
Lung_Opacity-4845.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4845.png,Lung_Opacity/masks/Lung_Opacity-4845.png,True,G07675,train,a85c9673c2a1794d4d69d4c1137d270c,917b461a39bb2ba4
Lung_Opacity-4846.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4846.png,Lung_Opacity/masks/Lung_Opacity-4846.png,True,G07676,train,91b0d345a196c7200643e6d233975262,913a0e5f5f4f45c0
Lung_Opacity-4847.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4847.png,Lung_Opacity/masks/Lung_Opacity-4847.png,True,G07677,train,dafefc0c6aaf0ab3f52616272e3a5e6f,d52d4f460b0a2b5f
Lung_Opacity-4848.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4848.png,Lung_Opacity/masks/Lung_Opacity-4848.png,True,G07678,train,37a8b739170879b10905150f25146dfb,d91a561a1c5b1f1e
Lung_Opacity-4849.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4849.png,Lung_Opacity/masks/Lung_Opacity-4849.png,True,G07679,train,b12a6e2751294a2c4d8a550213f2e294,887b85d69e85c63c
Lung_Opacity-485.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-485.png,Lung_Opacity/masks/Lung_Opacity-485.png,True,G07680,train,8db30e25a178e722b4ac1fdda02b273c,887947869eb8eae2
Lung_Opacity-4850.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4850.png,Lung_Opacity/masks/Lung_Opacity-4850.png,True,G07681,train,9e6f4ed2864ce1e13d0b742fdc6ffaf8,c66fc10e2f161f98
Lung_Opacity-4851.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4851.png,Lung_Opacity/masks/Lung_Opacity-4851.png,True,G07682,train,b703f9113c627ce9d233a51d9fac00a0,c03c2fc73d896cb4
Lung_Opacity-4852.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4852.png,Lung_Opacity/masks/Lung_Opacity-4852.png,True,G07683,train,fd63c4888df535b2cfd6e41fb09b0b82,916f9618491d5f0f
Lung_Opacity-4853.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4853.png,Lung_Opacity/masks/Lung_Opacity-4853.png,True,G07684,train,290b6e702c9e2a959a52d46d0f9dd0cf,dc3d42451f0a2dbd
Lung_Opacity-4854.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4854.png,Lung_Opacity/masks/Lung_Opacity-4854.png,True,G07685,train,10177672c838a0d77811553cdf9be2e4,917e07560f4d195d
Lung_Opacity-4855.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4855.png,Lung_Opacity/masks/Lung_Opacity-4855.png,True,G07686,train,dfd1bb712b395a288fad94c9159f7565,917b471a59edb841
Lung_Opacity-4856.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4856.png,Lung_Opacity/masks/Lung_Opacity-4856.png,True,G07687,val,c598a78ad0ce85c275704ec2897f92ad,985a861e5c792ba7
Lung_Opacity-4857.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4857.png,Lung_Opacity/masks/Lung_Opacity-4857.png,True,G07688,train,2c58174f404a7c6747bb66a71ab37843,917a071a5b793a69
Lung_Opacity-4858.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4858.png,Lung_Opacity/masks/Lung_Opacity-4858.png,True,G07689,val,d3d3087d287302b4fcc40e7d3981f831,cc2dd30f4f484e0f
Lung_Opacity-4859.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4859.png,Lung_Opacity/masks/Lung_Opacity-4859.png,True,G07690,val,9a054197027d0b90288e2e1cf7c252d2,917a80077879795f
Lung_Opacity-486.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-486.png,Lung_Opacity/masks/Lung_Opacity-486.png,True,G07691,test,79ccd61e749e1b73fef8cdf27121f863,82631d147a7a7b59
Lung_Opacity-4860.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4860.png,Lung_Opacity/masks/Lung_Opacity-4860.png,True,G07692,test,9c011b52bc3b80b8097533f1126b681d,9c47034d6c7ce1e3
Lung_Opacity-4861.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4861.png,Lung_Opacity/masks/Lung_Opacity-4861.png,True,G07693,test,a4cac88e8008e8a26bb7db9efb35a897,cc3c4f0c506fb8e5
Lung_Opacity-4862.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4862.png,Lung_Opacity/masks/Lung_Opacity-4862.png,True,G07694,train,28b257f49ebea51b3b0abf83fc994577,d57e434f0f4a0d0e
Lung_Opacity-4863.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4863.png,Lung_Opacity/masks/Lung_Opacity-4863.png,True,G07695,train,81356da43088b3479f63ac231590184b,d62d4a0e0f0f0f1f
Lung_Opacity-4864.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4864.png,Lung_Opacity/masks/Lung_Opacity-4864.png,True,G07696,train,bbd5402017e8df530fdf5bf8eb666e19,817a864a595d53fa
Lung_Opacity-4865.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4865.png,Lung_Opacity/masks/Lung_Opacity-4865.png,True,G07697,train,18fd6936d265136337f698d14a2d45b1,d92cc61fc4571b45
Lung_Opacity-4866.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4866.png,Lung_Opacity/masks/Lung_Opacity-4866.png,True,G07698,val,e911d293ad9e40c638f69560eba703f9,d42e01511a6e6e7f
Lung_Opacity-4867.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4867.png,Lung_Opacity/masks/Lung_Opacity-4867.png,True,G07699,val,1204585fcd65572eed659df76b79bbe8,d52883171f5f1b1a
Lung_Opacity-4868.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4868.png,Lung_Opacity/masks/Lung_Opacity-4868.png,True,G07700,train,4820e67a7b9c842c71919f1c28de5573,c42d424b0b4f6e7e
Lung_Opacity-4869.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4869.png,Lung_Opacity/masks/Lung_Opacity-4869.png,True,G07701,train,4b84ca617037e059ab3b0926ed565b17,c03d4a0b0b0bbfaf
Lung_Opacity-487.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-487.png,Lung_Opacity/masks/Lung_Opacity-487.png,True,G07702,val,81e8f13f99dc95d33ec89adaf9f0c408,c43d5b1a4b1b4b4b
Lung_Opacity-4870.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4870.png,Lung_Opacity/masks/Lung_Opacity-4870.png,True,G07703,test,fc4fe336fa4a01ba74aea30a6ee2d72f,c938b64b1d59d0e5
Lung_Opacity-4871.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4871.png,Lung_Opacity/masks/Lung_Opacity-4871.png,True,G07704,train,6fe692d6527608d52d406c442f4047c3,8179cf861c188bbf
Lung_Opacity-4872.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4872.png,Lung_Opacity/masks/Lung_Opacity-4872.png,True,G07705,train,a9534c43b8d5f2693ad228bfe2dfd5fc,8139d6961859fb4b
Lung_Opacity-4873.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4873.png,Lung_Opacity/masks/Lung_Opacity-4873.png,True,G07706,train,4116fe942c87edd684d13fda50206801,cc6c57c34f1c7849
Lung_Opacity-4874.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4874.png,Lung_Opacity/masks/Lung_Opacity-4874.png,True,G07707,train,55148e8765f2b86d30d94a63c9a5aca5,d909461b196d6db9
Lung_Opacity-4875.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4875.png,Lung_Opacity/masks/Lung_Opacity-4875.png,True,G07708,test,107f6d595a5ce504189ca997ccfa1fdb,9178c7161a4d7a79
Lung_Opacity-4876.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4876.png,Lung_Opacity/masks/Lung_Opacity-4876.png,True,G07709,train,c43ca9e50ed8925b7f988dd88b112d80,d17b47061e0b6b36
Lung_Opacity-4877.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4877.png,Lung_Opacity/masks/Lung_Opacity-4877.png,True,G07710,train,ec270d631848339da03e00b0f02aad42,cc4c534b4f4d0e1f
Lung_Opacity-4878.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4878.png,Lung_Opacity/masks/Lung_Opacity-4878.png,True,G07711,test,03f63859387750960d64878565ba9e3f,c93f071e1c4b1c5d
Lung_Opacity-4879.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4879.png,Lung_Opacity/masks/Lung_Opacity-4879.png,True,G07712,train,f15df3a1efff1940fb7aa8a3980a7800,9858461e1a1f3f3e
Lung_Opacity-488.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-488.png,Lung_Opacity/masks/Lung_Opacity-488.png,True,G07713,test,52264876df0fc72e6714c2b0d4a96f28,856e47c70a4d4d4f
Lung_Opacity-4880.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4880.png,Lung_Opacity/masks/Lung_Opacity-4880.png,True,G07714,train,8bdea5e208308f6bfe4eb2d636dc5186,9979a71e1a051b3d
Lung_Opacity-4881.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4881.png,Lung_Opacity/masks/Lung_Opacity-4881.png,True,G07715,train,321596bed87d394775b87ac2174e5f4b,c82f871b1c4c1f5d
Lung_Opacity-4882.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4882.png,Lung_Opacity/masks/Lung_Opacity-4882.png,True,G07716,train,4e96ad688c0f618a009c0677088d07b4,c02acf5a590e1d3f
Lung_Opacity-4883.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4883.png,Lung_Opacity/masks/Lung_Opacity-4883.png,True,G07717,train,28ec0e759b1f4d803b6d475fb056367d,913943961fcd1b3c
Lung_Opacity-4884.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4884.png,Lung_Opacity/masks/Lung_Opacity-4884.png,True,G07718,train,cdc5c097076cf401a93feba39a0368ce,956806165a5d5d5f
Lung_Opacity-4885.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4885.png,Lung_Opacity/masks/Lung_Opacity-4885.png,True,G07719,train,1c80b8b0b11e0a030c74a64c097523a8,913e4e0c4d7e7c4c
Lung_Opacity-4886.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4886.png,Lung_Opacity/masks/Lung_Opacity-4886.png,True,G07720,train,e2ad3197f819939d731659acdb41f7db,813b57045f1f5e1a
Lung_Opacity-4887.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4887.png,Lung_Opacity/masks/Lung_Opacity-4887.png,True,G07721,train,6db06ccc5803aff2fa6730d7cf45c760,dc2e435b015f4d4e
Lung_Opacity-4888.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4888.png,Lung_Opacity/masks/Lung_Opacity-4888.png,True,G07722,train,59ac642a47fa36d00789e447ce1ca9d5,89780c5efae9e8a1
Lung_Opacity-4889.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4889.png,Lung_Opacity/masks/Lung_Opacity-4889.png,True,G07723,train,fd1be07221dee876802e952150f8c457,d722412e2e75f4d1
Lung_Opacity-489.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-489.png,Lung_Opacity/masks/Lung_Opacity-489.png,True,G07724,train,1858e9046fec60afba723dda18d37359,c13f025f1f1a5d1a
Lung_Opacity-4890.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4890.png,Lung_Opacity/masks/Lung_Opacity-4890.png,True,G07725,train,0cd1f1e37dd66b9b9f3916b3ac2c7884,cc2bd21f16c30f1e
Lung_Opacity-4891.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4891.png,Lung_Opacity/masks/Lung_Opacity-4891.png,True,G07726,train,5479e443a06b5a3530d313d0bde8bfe3,9953565a1b1a0f3e
Lung_Opacity-4892.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4892.png,Lung_Opacity/masks/Lung_Opacity-4892.png,True,G07727,train,6904ab6a6ef89a4818f937e0908ace5b,907f46141a5b7b59
Lung_Opacity-4893.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4893.png,Lung_Opacity/masks/Lung_Opacity-4893.png,True,G07728,val,4265648fd96762fa86229968442f9111,9169071e1b69bcb9
Lung_Opacity-4894.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4894.png,Lung_Opacity/masks/Lung_Opacity-4894.png,True,G07729,val,664ab069406f1a7be9be5dfb6c457a80,d02f42171a1f5b1f
Lung_Opacity-4895.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4895.png,Lung_Opacity/masks/Lung_Opacity-4895.png,True,G07730,val,08b044f32aabff6a991fdb9cf763044c,c17b50180f0faf1f
Lung_Opacity-4896.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4896.png,Lung_Opacity/masks/Lung_Opacity-4896.png,True,G07731,test,c3e6c1a7211218b7c539c16eb4b5dd15,882e070f4d4f4f2f
Lung_Opacity-4897.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4897.png,Lung_Opacity/masks/Lung_Opacity-4897.png,True,G07732,test,a53aab2248a7917efc858613c2d84bd1,dc28410f4f0e3f3d
Lung_Opacity-4898.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4898.png,Lung_Opacity/masks/Lung_Opacity-4898.png,True,G07733,train,239ff40e2a4a2b613c373f76b094a044,887907161e5a5f5d
Lung_Opacity-4899.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4899.png,Lung_Opacity/masks/Lung_Opacity-4899.png,True,G07734,train,266c0b2871ef1d64b2d33c46db49d421,897d061e5f1a185f
Lung_Opacity-49.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-49.png,Lung_Opacity/masks/Lung_Opacity-49.png,True,G07735,train,352b62c51059d34440a4628e6000cce5,817817421b1dcf5f
Lung_Opacity-490.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-490.png,Lung_Opacity/masks/Lung_Opacity-490.png,True,G07736,train,49c73a3308c18996805acb726f1a779e,ce3d594e4b0d2e2c
Lung_Opacity-4900.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4900.png,Lung_Opacity/masks/Lung_Opacity-4900.png,True,G07737,train,bbd2a837e390332f44e70b25d53fe679,857cad150a787d69
Lung_Opacity-4901.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4901.png,Lung_Opacity/masks/Lung_Opacity-4901.png,True,G07738,train,b907d39a93a788241d9b7979d0042957,d40a4f0a5eabf0bc
Lung_Opacity-4902.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4902.png,Lung_Opacity/masks/Lung_Opacity-4902.png,True,G07739,train,2f824f2108ea761e94ba7ba12b9c2e3c,807b0c1a5b4fe73a
Lung_Opacity-4903.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4903.png,Lung_Opacity/masks/Lung_Opacity-4903.png,True,G07740,train,604a9e0747cb405189286b587bd46063,d42d5b0a2a3cb2b7
Lung_Opacity-4904.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4904.png,Lung_Opacity/masks/Lung_Opacity-4904.png,True,G07741,val,965a2c6fc35a57027925a0f00524fbfd,c03c0fc71b19d95b
Lung_Opacity-4905.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4905.png,Lung_Opacity/masks/Lung_Opacity-4905.png,True,G07742,train,fae9c896fce4e90c3eb6bd32008af3c8,c53c07021b7c6d7e
Lung_Opacity-4906.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4906.png,Lung_Opacity/masks/Lung_Opacity-4906.png,True,G07743,train,c1675ae835e98099b18dfd057ea22cb6,c17a15561e14575f
Lung_Opacity-4907.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4907.png,Lung_Opacity/masks/Lung_Opacity-4907.png,True,G07744,train,546eb81e664a0f1012b659f75f461879,d07f405a1b093f1f
Lung_Opacity-4908.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4908.png,Lung_Opacity/masks/Lung_Opacity-4908.png,True,G07745,train,ea3619cc442164fb98f5fdf25b4093ba,9b099c5e5b485bc3
Lung_Opacity-4909.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4909.png,Lung_Opacity/masks/Lung_Opacity-4909.png,True,G07746,test,9bfad4ac5863d18e7e71113a3698e769,d46f432a2e2e272e
Lung_Opacity-491.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-491.png,Lung_Opacity/masks/Lung_Opacity-491.png,True,G07747,test,27f130ee9a29605b13eb684da4ccb60d,cc3d430ec82df0e7
Lung_Opacity-4910.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4910.png,Lung_Opacity/masks/Lung_Opacity-4910.png,True,G07748,train,07cb88cfcdcc1e55eae58569a401a69d,d829d7525a5a5b49
Lung_Opacity-4911.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4911.png,Lung_Opacity/masks/Lung_Opacity-4911.png,True,G07749,test,035cc70264399206f177cad980641c36,8a2f434e4e6e6a6a
Lung_Opacity-4912.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4912.png,Lung_Opacity/masks/Lung_Opacity-4912.png,True,G07750,train,14f2146dbcb51c0cf23495bb5786e0fa,817f861a1b4d5b5c
Lung_Opacity-4913.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4913.png,Lung_Opacity/masks/Lung_Opacity-4913.png,True,G07751,train,c758d0b681555bc55738dbf96da72ef1,916c0e5b0d7d68ec
Lung_Opacity-4914.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4914.png,Lung_Opacity/masks/Lung_Opacity-4914.png,True,G07752,val,2efafc570acdaba9fef37555202b37fe,817c0e1f7b7968c8
Lung_Opacity-4915.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4915.png,Lung_Opacity/masks/Lung_Opacity-4915.png,True,G07753,val,a5e2c7df9118b9d4be227c60d75c19db,d12f16923bc439d9
Lung_Opacity-4916.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4916.png,Lung_Opacity/masks/Lung_Opacity-4916.png,True,G07754,val,2ceac37d4e819bdde4b721f8404f5007,880e8a5b697d7ae2
Lung_Opacity-4917.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4917.png,Lung_Opacity/masks/Lung_Opacity-4917.png,True,G07755,train,ecc9dd42a73d2594c780cf05aa376916,d83802475e5f5a4f
Lung_Opacity-4918.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4918.png,Lung_Opacity/masks/Lung_Opacity-4918.png,True,G07756,train,3a4f25a0e3ffd0fae1d885699a4136b3,807e07163e1f585f
Lung_Opacity-4919.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4919.png,Lung_Opacity/masks/Lung_Opacity-4919.png,True,G07757,train,eaf4b1061002d5522ea09295d10d245e,983d4a160e4b1f7d
Lung_Opacity-492.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-492.png,Lung_Opacity/masks/Lung_Opacity-492.png,True,G07758,train,1da4f4d027327b120903a23c1c7edbad,d52a934d45564d57
Lung_Opacity-4920.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4920.png,Lung_Opacity/masks/Lung_Opacity-4920.png,True,G07759,train,403e1b9c0cb22b836e295d6b2a3b8ffb,c92c165a69eb78e1
Lung_Opacity-4921.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4921.png,Lung_Opacity/masks/Lung_Opacity-4921.png,True,G07760,train,b06f7e01e428560e5ee1ac79c179f9f2,c87bd75e14125b51
Lung_Opacity-4922.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4922.png,Lung_Opacity/masks/Lung_Opacity-4922.png,True,G07761,train,06e7c489a3a71d672cf86f92a4304308,9839d20f5d1b1b1b
Lung_Opacity-4923.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4923.png,Lung_Opacity/masks/Lung_Opacity-4923.png,True,G07762,train,7d1523063ea9bc22488a8795222b2e72,d928161b59393d2f
Lung_Opacity-4924.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4924.png,Lung_Opacity/masks/Lung_Opacity-4924.png,True,G07763,train,b261c4c35200d6fd4873840a7743a941,8b7a141f7a5e3838
Lung_Opacity-4925.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4925.png,Lung_Opacity/masks/Lung_Opacity-4925.png,True,G07764,val,c6c26627cc417a91efcc52a0574e25e8,8c3dc30bc55b1d4d
Lung_Opacity-4926.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4926.png,Lung_Opacity/masks/Lung_Opacity-4926.png,True,G07765,train,71e03f3e484324fb6da1a3f6cb1b7df0,c03bd6171e1e1b39
Lung_Opacity-4927.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4927.png,Lung_Opacity/masks/Lung_Opacity-4927.png,True,G07766,train,25873164180d472f7aa23014dc0a6929,912bc6109e17bbd6
Lung_Opacity-4928.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4928.png,Lung_Opacity/masks/Lung_Opacity-4928.png,True,G07767,val,d363edcd9c6ebefd360a155a842036d7,cc3c1f1b585e6c29
Lung_Opacity-4929.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4929.png,Lung_Opacity/masks/Lung_Opacity-4929.png,True,G07768,test,efc16fdbae3984ccfb8c3b1a8fed8d55,d93995461e1e5b0b
Lung_Opacity-493.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-493.png,Lung_Opacity/masks/Lung_Opacity-493.png,True,G07769,train,bc46f249d1b71264f36be8a4fa4c17a0,d42f030d0e3e3e73
Lung_Opacity-4930.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4930.png,Lung_Opacity/masks/Lung_Opacity-4930.png,True,G07770,train,aa7b57078e1b38737a6c645d28e272ae,d93d420e1c69d53d
Lung_Opacity-4931.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4931.png,Lung_Opacity/masks/Lung_Opacity-4931.png,True,G07771,test,1644d89daa1ec3058bc623d9d6580a8b,dc6b537ce0f0e0c2
Lung_Opacity-4932.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4932.png,Lung_Opacity/masks/Lung_Opacity-4932.png,True,G07772,test,ba8f39e7bf9b3ce1365335be5a5d60be,9128065b4c7e79ed
Lung_Opacity-4933.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4933.png,Lung_Opacity/masks/Lung_Opacity-4933.png,True,G07773,train,85312646ef6b2f52a3747784f1722145,8139121e5e5b9e1f
Lung_Opacity-4934.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4934.png,Lung_Opacity/masks/Lung_Opacity-4934.png,True,G07774,train,1034b3cbeca87010a6e60c309e944a02,c938c74b1e436d3c
Lung_Opacity-4935.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4935.png,Lung_Opacity/masks/Lung_Opacity-4935.png,True,G07775,train,2f94e9256544c41dc7ab52813975c3e5,d42f0b475d4d0f0e
Lung_Opacity-4936.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4936.png,Lung_Opacity/masks/Lung_Opacity-4936.png,True,G07776,train,e04961e0697a0f0318652a54d130c372,dd2e510b1f4d095d
Lung_Opacity-4937.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4937.png,Lung_Opacity/masks/Lung_Opacity-4937.png,True,G07777,train,facd383e518a83c6bf0be5ced580ff84,d40c29439f5a6b6b
Lung_Opacity-4938.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4938.png,Lung_Opacity/masks/Lung_Opacity-4938.png,True,G07778,train,0d570fe652f751639c4144d623ef672e,d43e524b0f464f1d
Lung_Opacity-4939.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4939.png,Lung_Opacity/masks/Lung_Opacity-4939.png,True,G07779,test,be53322ac0d1558184844a8650ae9a52,881b1618595f5afe
Lung_Opacity-494.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-494.png,Lung_Opacity/masks/Lung_Opacity-494.png,True,G07780,train,4d0720c8286c95c7336125b1afdef181,d95bc650c347d0c7
Lung_Opacity-4940.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4940.png,Lung_Opacity/masks/Lung_Opacity-4940.png,True,G07781,train,cd532d26debe2d0299f4c7e128f8a3a5,d13b421e597b3938
Lung_Opacity-4941.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4941.png,Lung_Opacity/masks/Lung_Opacity-4941.png,True,G07782,train,2d246081507cbbb8ed8fcc049b3643ff,cc07470f4a4f6a6b
Lung_Opacity-4942.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4942.png,Lung_Opacity/masks/Lung_Opacity-4942.png,True,G07783,test,5a3e66b3f0c2f5337a5ca8f7a521d7a4,817d061c584f6f79
Lung_Opacity-4943.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4943.png,Lung_Opacity/masks/Lung_Opacity-4943.png,True,G07784,test,cfcd39aa2e01f75d231f0f93864d0902,d52a410f8eaaabab
Lung_Opacity-4944.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4944.png,Lung_Opacity/masks/Lung_Opacity-4944.png,True,G07785,val,5cab09dfc2b43780cd65a5fea9884bc7,dd0c46421d2d6dbe
Lung_Opacity-4945.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4945.png,Lung_Opacity/masks/Lung_Opacity-4945.png,True,G07786,train,d205f767c8c931dc766f91b21143aa06,d06d46140b6f3d3d
Lung_Opacity-4946.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4946.png,Lung_Opacity/masks/Lung_Opacity-4946.png,True,G07787,test,9e182182a462d6adfcc3d0ab1aff4060,d0395b5a1a39f097
Lung_Opacity-4947.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4947.png,Lung_Opacity/masks/Lung_Opacity-4947.png,True,G07788,val,a3feb43866013a77198d09b89d0cba72,d42ad75f58482c6e
Lung_Opacity-4948.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4948.png,Lung_Opacity/masks/Lung_Opacity-4948.png,True,G07789,train,1e9d20ef6e10930ea867c628a0ca34c3,903d430f1e5d693d
Lung_Opacity-4949.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4949.png,Lung_Opacity/masks/Lung_Opacity-4949.png,True,G07790,train,0973e451405045320012bea10330f2a2,c83d4b1b2c3c3cad
Lung_Opacity-495.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-495.png,Lung_Opacity/masks/Lung_Opacity-495.png,True,G07791,val,2f5e6bce44d4b63daeaded6cf6042c22,947e03474e6e6a6c
Lung_Opacity-4950.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4950.png,Lung_Opacity/masks/Lung_Opacity-4950.png,True,G07792,test,5b94f65194888b72402a5fba036adb32,c41f0f1c4c5d5cda
Lung_Opacity-4951.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4951.png,Lung_Opacity/masks/Lung_Opacity-4951.png,True,G07793,train,76f9778617bd69325fa611f6ce99bb5d,d434f96a684a6c6e
Lung_Opacity-4952.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4952.png,Lung_Opacity/masks/Lung_Opacity-4952.png,True,G07794,train,207a289f45a584a03107c7d01b7bce69,dd2f8616184d5e78
Lung_Opacity-4953.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4953.png,Lung_Opacity/masks/Lung_Opacity-4953.png,True,G07795,train,741af3d9c32b551d8e69a1fc905c6b75,887b05d67ae167a4
Lung_Opacity-4954.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4954.png,Lung_Opacity/masks/Lung_Opacity-4954.png,True,G07796,train,796cb7483df29870b15cd6129375904b,91605f5a1b5b1b3a
Lung_Opacity-4955.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4955.png,Lung_Opacity/masks/Lung_Opacity-4955.png,True,G07797,train,e9b4249216b82c24fc47fdd7b2ab8356,982ec21b1c1e7e3e
Lung_Opacity-4956.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4956.png,Lung_Opacity/masks/Lung_Opacity-4956.png,True,G07798,train,01b2dacfeb05f714e44730c03d68aafc,cd28160e1e5f5a5d
Lung_Opacity-4957.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4957.png,Lung_Opacity/masks/Lung_Opacity-4957.png,True,G07799,train,efa11d0613564d952612464a7e945acd,d13d46c31e19cd8d
Lung_Opacity-4958.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4958.png,Lung_Opacity/masks/Lung_Opacity-4958.png,True,G07800,train,a2753ff57601c16045aba3db8b0d4383,9139ce1e185b4b79
Lung_Opacity-4959.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4959.png,Lung_Opacity/masks/Lung_Opacity-4959.png,True,G07801,train,8bfc976e9d2cc9c9ca57a19372236137,c46dd35a1f0b093d
Lung_Opacity-496.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-496.png,Lung_Opacity/masks/Lung_Opacity-496.png,True,G07802,train,923139e01002cb10d28912e0809e1867,907fc71a5a5b1c1a
Lung_Opacity-4960.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4960.png,Lung_Opacity/masks/Lung_Opacity-4960.png,True,G07803,train,61022831ee18b6e108cd76a24a7bdafc,c0fd0d0c5e0d5f1d
Lung_Opacity-4961.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4961.png,Lung_Opacity/masks/Lung_Opacity-4961.png,True,G07804,train,27e11bb430668e853d1699f9ead6ed24,c47c0f4ab5f0e5c2
Lung_Opacity-4962.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4962.png,Lung_Opacity/masks/Lung_Opacity-4962.png,True,G07805,train,a96c1e09b027d7a5241e170fcd3ec460,802f0b5e1a1b7f1e
Lung_Opacity-4963.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4963.png,Lung_Opacity/masks/Lung_Opacity-4963.png,True,G07806,train,7f2be3cea91b1853ea381a146941e5de,d5380d151d5c3c7d
Lung_Opacity-4964.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4964.png,Lung_Opacity/masks/Lung_Opacity-4964.png,True,G07807,train,a4079f2e0c81a3d9e23068d53208c04b,c03e3b622972f3c3
Lung_Opacity-4965.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4965.png,Lung_Opacity/masks/Lung_Opacity-4965.png,True,G07808,train,062124164c59f765f9beffa652a4ddc2,817f82465d4d0d5f
Lung_Opacity-4966.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4966.png,Lung_Opacity/masks/Lung_Opacity-4966.png,True,G07809,val,ac982b4ffbc03fcbb33bb9d0bfcdb958,d82d5f0a0c1f2d9d
Lung_Opacity-4967.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4967.png,Lung_Opacity/masks/Lung_Opacity-4967.png,True,G07810,val,c7e14c5d351f9e1c5564c539358134e1,897fc6125a4a4f59
Lung_Opacity-4968.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4968.png,Lung_Opacity/masks/Lung_Opacity-4968.png,True,G07811,val,3f1fff88ebb1c8a273297bbd888414ef,d07d431b1a0fad2d
Lung_Opacity-4969.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4969.png,Lung_Opacity/masks/Lung_Opacity-4969.png,True,G07812,test,791d4d594ca5794c23f9c159937969c1,c2273d789bc7e260
Lung_Opacity-497.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-497.png,Lung_Opacity/masks/Lung_Opacity-497.png,True,G07813,test,d539319f3daf66ff6fa321716cf0e8f4,9978061a597979e9
Lung_Opacity-4970.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4970.png,Lung_Opacity/masks/Lung_Opacity-4970.png,True,G07814,train,fccc0786ee9deec6758001a8612d5184,886947971fcb2d98
Lung_Opacity-4971.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4971.png,Lung_Opacity/masks/Lung_Opacity-4971.png,True,G07815,train,1f3adaf8987a9fa70df6bc5f0d15fd84,dc3d4f4b0e0c2f2c
Lung_Opacity-4972.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4972.png,Lung_Opacity/masks/Lung_Opacity-4972.png,True,G07816,val,55e684f89f55f0d10f9572b4bc3fcce8,c03dc3d55b4b4b1a
Lung_Opacity-4973.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4973.png,Lung_Opacity/masks/Lung_Opacity-4973.png,True,G07817,train,b51bd59dfef0941df8dc31c569b0ec1a,c03f52421b29fb2f
Lung_Opacity-4974.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4974.png,Lung_Opacity/masks/Lung_Opacity-4974.png,True,G07818,train,74d6349f8a548cdd486a60e2f33dd5ce,d42dd741d753505a
Lung_Opacity-4975.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4975.png,Lung_Opacity/masks/Lung_Opacity-4975.png,True,G07819,train,70fa763b64db7544b81535c0a6fe90f4,c53c920b2d377732
Lung_Opacity-4976.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4976.png,Lung_Opacity/masks/Lung_Opacity-4976.png,True,G07820,train,946d06eb4f31f45e4994f913804bbb16,907f04183b7b39bc
Lung_Opacity-4977.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4977.png,Lung_Opacity/masks/Lung_Opacity-4977.png,True,G07821,train,0370d471d6ebba7453737fd9e7210a2b,8179471e1e5a1b5b
Lung_Opacity-4978.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4978.png,Lung_Opacity/masks/Lung_Opacity-4978.png,True,G07822,train,94d8069eaae58e122f61941c5b0a5a5c,9178071b1f5d3978
Lung_Opacity-4979.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4979.png,Lung_Opacity/masks/Lung_Opacity-4979.png,True,G07823,train,7dfec129b71346f568821bdf8568db2a,9d68861e1d5d5959
Lung_Opacity-498.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-498.png,Lung_Opacity/masks/Lung_Opacity-498.png,True,G07824,train,c75fb995dbcf560e97fc3ba2b4bb858c,856c4c0e6dcee6d2
Lung_Opacity-4980.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4980.png,Lung_Opacity/masks/Lung_Opacity-4980.png,True,G07825,val,dfb06765d3d4201774e77cd89e122ef2,956a30668d995b6e
Lung_Opacity-4981.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4981.png,Lung_Opacity/masks/Lung_Opacity-4981.png,True,G07826,test,3244fc293f9649fa864795aab71a25e6,9568045f3a5d3e5c
Lung_Opacity-4982.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4982.png,Lung_Opacity/masks/Lung_Opacity-4982.png,True,G07827,train,f08b4cec74da016137bbdcde1bfc96ec,8079aa5f1b585da9
Lung_Opacity-4983.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4983.png,Lung_Opacity/masks/Lung_Opacity-4983.png,True,G07828,test,c0acb73503e9d77d5570aaeca0bc0bde,dd2f834b1d0d389a
Lung_Opacity-4984.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4984.png,Lung_Opacity/masks/Lung_Opacity-4984.png,True,G07829,test,cfbff810fd765f44579a74581101ea95,d13b02fe121f131f
Lung_Opacity-4985.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4985.png,Lung_Opacity/masks/Lung_Opacity-4985.png,True,G07830,train,b73f0100757e64e3f7c6120642bde936,d03c2f43d31b0d6d
Lung_Opacity-4986.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4986.png,Lung_Opacity/masks/Lung_Opacity-4986.png,True,G07831,train,f12486eb4b5c6b1eb9adf4d3febcdd91,8077151b7b3e6a29
Lung_Opacity-4987.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4987.png,Lung_Opacity/masks/Lung_Opacity-4987.png,True,G07832,train,fbd69243050c151ac6e84b993c6f88f2,993dc61a594d383b
Lung_Opacity-4988.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4988.png,Lung_Opacity/masks/Lung_Opacity-4988.png,True,G07833,val,85c8b04c7f63d44b6204063534c63fef,c52e470b4d4e4e6e
Lung_Opacity-4989.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4989.png,Lung_Opacity/masks/Lung_Opacity-4989.png,True,G07834,val,c2f56ea9524a59ff6e289c0bb7b9e710,885d045f1d7c6cea
Lung_Opacity-499.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-499.png,Lung_Opacity/masks/Lung_Opacity-499.png,True,G07835,train,a0e654cd386117cd70ba1574e6dc3506,c27d0f2c076fc5c2
Lung_Opacity-4990.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4990.png,Lung_Opacity/masks/Lung_Opacity-4990.png,True,G07836,train,327e91e6fe66fe0969f2a05b821fc269,8178075b1f5e3a6a
Lung_Opacity-4991.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4991.png,Lung_Opacity/masks/Lung_Opacity-4991.png,True,G07837,train,a5af4b5ee2253f3bbe29238c020e30af,91681f1e1a5b7978
Lung_Opacity-4992.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4992.png,Lung_Opacity/masks/Lung_Opacity-4992.png,True,G07838,train,29d40af375087a57412db463a5e0f0aa,fb781c58597b5908
Lung_Opacity-4993.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4993.png,Lung_Opacity/masks/Lung_Opacity-4993.png,True,G07839,train,9736d607aefd9aba92d3031cd2430c70,c03d16434f3c7c6b
Lung_Opacity-4994.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4994.png,Lung_Opacity/masks/Lung_Opacity-4994.png,True,G07840,train,5e24acc30c5547f717dc6b943dc318f4,8c6cd353555d0d4d
Lung_Opacity-4995.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4995.png,Lung_Opacity/masks/Lung_Opacity-4995.png,True,G07841,train,a83ec4c2d23fdd07ea61b17fbfdc392e,d02d7f12162565e7
Lung_Opacity-4996.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4996.png,Lung_Opacity/masks/Lung_Opacity-4996.png,True,G07842,train,06f206299efabed7b9586e3f25ee2abe,d53d4a0a083f6f4e
Lung_Opacity-4997.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4997.png,Lung_Opacity/masks/Lung_Opacity-4997.png,True,G07843,train,fa19b9ad47a77598593ad583d7a0003d,c03d1b183e32b7b3
Lung_Opacity-4998.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4998.png,Lung_Opacity/masks/Lung_Opacity-4998.png,True,G07844,train,a2ff939bd8bf17df1a83fecf5aeb755d,81795e4658597a3b
Lung_Opacity-4999.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-4999.png,Lung_Opacity/masks/Lung_Opacity-4999.png,True,G07845,train,55e7289874ebd5a25e56c17514e35781,93608d1b1a5b7b5a
Lung_Opacity-5.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5.png,Lung_Opacity/masks/Lung_Opacity-5.png,True,G07846,val,f70d3d33fadaa31cd340a63ed3cc12cd,99695c1f5be9a049
Lung_Opacity-50.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-50.png,Lung_Opacity/masks/Lung_Opacity-50.png,True,G07847,train,5311a1f7761b647bfafc24488f614a22,917847431b4b4f1f
Lung_Opacity-500.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-500.png,Lung_Opacity/masks/Lung_Opacity-500.png,True,G07848,train,38ad480ce1d7880a72410f29b93590d8,905a861e7a39fea4
Lung_Opacity-5000.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5000.png,Lung_Opacity/masks/Lung_Opacity-5000.png,True,G07849,val,53dc1caad11be712d02d5e0fd6581cb9,ca3f13f1920bd21b
Lung_Opacity-5001.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5001.png,Lung_Opacity/masks/Lung_Opacity-5001.png,True,G07850,test,c066ccf214a0d2b07c818710061e07ff,dc3d5a81cb8d4e4c
Lung_Opacity-5002.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5002.png,Lung_Opacity/masks/Lung_Opacity-5002.png,True,G07851,val,72b892ce4f3b814a1e8cc2c74491fe4e,8978165a585f7a78
Lung_Opacity-5003.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5003.png,Lung_Opacity/masks/Lung_Opacity-5003.png,True,G07852,train,dda6e281757150a38ca293d2e94737e9,c42fa70d05667ee4
Lung_Opacity-5004.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5004.png,Lung_Opacity/masks/Lung_Opacity-5004.png,True,G07853,test,7c0553614cb746ba319ee39bf7b3b3ac,8079d6161a4b6b3f
Lung_Opacity-5005.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5005.png,Lung_Opacity/masks/Lung_Opacity-5005.png,True,G07854,test,40e099563fb516764697c236748adc30,81ae8718870f3d6f
Lung_Opacity-5006.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5006.png,Lung_Opacity/masks/Lung_Opacity-5006.png,True,G07855,train,3735b37c26a4b3c4af48781e2c744f37,896c465e4929a7e7
Lung_Opacity-5007.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5007.png,Lung_Opacity/masks/Lung_Opacity-5007.png,True,G07856,train,44b96ecc6693232498dc8e8f4a6571d6,817bc642686beaea
Lung_Opacity-5008.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5008.png,Lung_Opacity/masks/Lung_Opacity-5008.png,True,G07857,train,dfb21df4f00ba18398c12517d4d36547,c03dcf4f18186d3d
Lung_Opacity-5009.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5009.png,Lung_Opacity/masks/Lung_Opacity-5009.png,True,G07858,train,f9282de2b5719ec61cbd85c7f6f4218a,9938175a79e1e187
Lung_Opacity-501.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-501.png,Lung_Opacity/masks/Lung_Opacity-501.png,True,G07859,train,6b7690971c26bc8f645db6ab1bf98280,c46d0a4c0e6cc7f7
Lung_Opacity-5010.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5010.png,Lung_Opacity/masks/Lung_Opacity-5010.png,True,G07860,train,a3528c0f4e5ad97e7c65ab2385a33cb5,913bcc1a58595d5e
Lung_Opacity-5011.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5011.png,Lung_Opacity/masks/Lung_Opacity-5011.png,True,G07861,train,8e9e6640d3a4287e8a416c9f4da5d138,da3d4d0e0c6c4f4b
Lung_Opacity-5012.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5012.png,Lung_Opacity/masks/Lung_Opacity-5012.png,True,G07862,train,00124ca6dc66ad0c499ad4efe3bfd9ea,d83d521a1f0f5a1d
Lung_Opacity-5013.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5013.png,Lung_Opacity/masks/Lung_Opacity-5013.png,True,G07863,train,348ae41f333af279ccd3c52901762bbc,d939c70e1c485a7d
Lung_Opacity-5014.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5014.png,Lung_Opacity/masks/Lung_Opacity-5014.png,True,G07864,val,979c5fa41666a5bfd94f9c957737ba35,8d2e071f4d5c686d
Lung_Opacity-5015.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5015.png,Lung_Opacity/masks/Lung_Opacity-5015.png,True,G07865,train,f316b6ab4346ec8ac92e58b8c9c1517d,817f87101b1f1f1b
Lung_Opacity-5016.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5016.png,Lung_Opacity/masks/Lung_Opacity-5016.png,True,G07866,val,291247462cb45020d0ae9be32ca70518,9c4fa6025b797c61
Lung_Opacity-5017.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5017.png,Lung_Opacity/masks/Lung_Opacity-5017.png,True,G07867,train,bcc01ee973098c26c435926071ec0e26,d62d510d0f6a2f0f
Lung_Opacity-5018.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5018.png,Lung_Opacity/masks/Lung_Opacity-5018.png,True,G07868,train,4636476c8cf2520b9c234b5691be458f,cc3d531b0e1c3e3a
Lung_Opacity-5019.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5019.png,Lung_Opacity/masks/Lung_Opacity-5019.png,True,G07869,train,5f76000f190f7a6afdfda257f80260d1,8a3d1a1abcbcb4b4
Lung_Opacity-502.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-502.png,Lung_Opacity/masks/Lung_Opacity-502.png,True,G07870,test,91e3fac104e211fce8ca1d005dd3be46,c42f430b2d2e36b7
Lung_Opacity-5020.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5020.png,Lung_Opacity/masks/Lung_Opacity-5020.png,True,G07871,train,47855cf78bc8e0423212dca5b74e282b,d62d414a2e256ded
Lung_Opacity-5021.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5021.png,Lung_Opacity/masks/Lung_Opacity-5021.png,True,G07872,test,bb45dfaa27ba9a45a7877710a7961ca5,996dc51e8a4f1e68
Lung_Opacity-5022.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5022.png,Lung_Opacity/masks/Lung_Opacity-5022.png,True,G07873,val,f24efd4b38cda5f3bab3c9e3fbeb3ee6,d03d0f081c2f6d7d
Lung_Opacity-5023.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5023.png,Lung_Opacity/masks/Lung_Opacity-5023.png,True,G07874,val,927e9333a0ed0cd1d7865b2ccf81c37c,817b5f510959295f
Lung_Opacity-5024.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5024.png,Lung_Opacity/masks/Lung_Opacity-5024.png,True,G07875,test,25d559a21d5e89365f02fa78a89fac15,906f471d0b1f0b4f
Lung_Opacity-5025.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5025.png,Lung_Opacity/masks/Lung_Opacity-5025.png,True,G07876,train,f3144aa08b68d4cfdeae847667d2d11c,92588c5bebc586ae
Lung_Opacity-5026.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5026.png,Lung_Opacity/masks/Lung_Opacity-5026.png,True,G07877,train,6637186c842ec78602aa508dc48ba974,d929d61a5d4b1c5c
Lung_Opacity-5027.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5027.png,Lung_Opacity/masks/Lung_Opacity-5027.png,True,G07878,test,7da799a6ee37d06b4f3756935bb976d6,d13b47121b5d1b1b
Lung_Opacity-5028.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5028.png,Lung_Opacity/masks/Lung_Opacity-5028.png,True,G07879,train,07cba84dc7699d7bd1433ad164b835e3,816e86168f1f385f
Lung_Opacity-5029.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5029.png,Lung_Opacity/masks/Lung_Opacity-5029.png,True,G07880,train,56c533a5b40e3a3ca45b7bbea2377cb4,812807961a5f5d7f
Lung_Opacity-503.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-503.png,Lung_Opacity/masks/Lung_Opacity-503.png,True,G07881,train,9caea5141c3a4b840ed84298e0c4b195,c93f464a18ede878
Lung_Opacity-5030.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5030.png,Lung_Opacity/masks/Lung_Opacity-5030.png,True,G07882,train,bcdb4dfa95232a3b2d6208144e6d326d,d43ec1db0b6ae00f
Lung_Opacity-5031.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5031.png,Lung_Opacity/masks/Lung_Opacity-5031.png,True,G07883,train,454c530cdc968df728ebc4bfda963a20,c13dd2d25b585959
Lung_Opacity-5032.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5032.png,Lung_Opacity/masks/Lung_Opacity-5032.png,True,G07884,train,d8c628f46135dc37b083c7ce7409f631,9179c6171929bcbc
Lung_Opacity-5033.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5033.png,Lung_Opacity/masks/Lung_Opacity-5033.png,True,G07885,val,b54051addc509726f64b05a35340b994,d03b5a16da1a9e1e
Lung_Opacity-5034.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5034.png,Lung_Opacity/masks/Lung_Opacity-5034.png,True,G07886,train,d787007e8de183740857bf9d0396db72,c56846616c78eafa
Lung_Opacity-5035.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5035.png,Lung_Opacity/masks/Lung_Opacity-5035.png,True,G07887,train,272e57ecc6f7b816a3652ad9ea573d3d,dd6ed21bc04a1d4b
Lung_Opacity-5036.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5036.png,Lung_Opacity/masks/Lung_Opacity-5036.png,True,G07888,train,62a57d534829d01c875bfea4ca7c57d1,957fc61a0b4f1a1a
Lung_Opacity-5037.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5037.png,Lung_Opacity/masks/Lung_Opacity-5037.png,True,G07889,train,8c970b99f4402722986e51fdcd915f46,917c0e587978a9ba
Lung_Opacity-5038.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5038.png,Lung_Opacity/masks/Lung_Opacity-5038.png,True,G07890,train,c6b2fef0425beb22a996151a75a73dad,d52e16c34adc6ed0
Lung_Opacity-5039.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5039.png,Lung_Opacity/masks/Lung_Opacity-5039.png,True,G07891,train,11a389ef9b27e6a76de7d2c332fef129,d42f4d0a0a0bbf9b
Lung_Opacity-504.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-504.png,Lung_Opacity/masks/Lung_Opacity-504.png,True,G07892,val,c9cda0dcef66335761280ee8094ff583,cd2dd0c35b4d0a5d
Lung_Opacity-5040.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5040.png,Lung_Opacity/masks/Lung_Opacity-5040.png,True,G07893,val,3f0400f5aa21a184f96cd730b1234228,d94d55171657464e
Lung_Opacity-5041.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5041.png,Lung_Opacity/masks/Lung_Opacity-5041.png,True,G07894,test,bb973694b8f89ee4057188fbce0f3ff2,d42dcbcb0c6c7c2c
Lung_Opacity-5042.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5042.png,Lung_Opacity/masks/Lung_Opacity-5042.png,True,G07895,train,6d9ae7cef2dcb326b7b0181c90a5d4a1,c93c164a7ce8e5d2
Lung_Opacity-5043.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5043.png,Lung_Opacity/masks/Lung_Opacity-5043.png,True,G07896,train,759bed795651d55772436f9893eb7da8,916b47141a497d3f
Lung_Opacity-5044.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5044.png,Lung_Opacity/masks/Lung_Opacity-5044.png,True,G07897,train,88ac0a02a71901b55d834af081127e91,81284b0fafa7c5c7
Lung_Opacity-5045.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5045.png,Lung_Opacity/masks/Lung_Opacity-5045.png,True,G07898,test,3959f8dba7953579268d1626ec7b2d18,887f06185b192f3f
Lung_Opacity-5046.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5046.png,Lung_Opacity/masks/Lung_Opacity-5046.png,True,G07899,train,415deec31d9b58e567acb26bbd6b9e3f,917f461a4f1a5d19
Lung_Opacity-5047.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5047.png,Lung_Opacity/masks/Lung_Opacity-5047.png,True,G07900,train,9a3aa214f8cc2df910d560362ed5a4ff,d42e535743431f5a
Lung_Opacity-5048.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5048.png,Lung_Opacity/masks/Lung_Opacity-5048.png,True,G07901,train,51be687870a2143464f7c43ed404dd79,c5786f966a996894
Lung_Opacity-5049.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5049.png,Lung_Opacity/masks/Lung_Opacity-5049.png,True,G07902,train,b917c528f592fe3d74fc04c141407f2b,842e0e5d4e4d6bcb
Lung_Opacity-505.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-505.png,Lung_Opacity/masks/Lung_Opacity-505.png,True,G07903,train,9fe3fc02b5da6662424ff4c76ad3fb05,997b041e5d5a5959
Lung_Opacity-5050.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5050.png,Lung_Opacity/masks/Lung_Opacity-5050.png,True,G07904,train,465b466c60a542988ebd78afda224930,89395a1b3e9296d6
Lung_Opacity-5051.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5051.png,Lung_Opacity/masks/Lung_Opacity-5051.png,True,G07905,train,3a0ebe66a0c27e029ba6e981e1a35ef8,c52d5a1a192de9ec
Lung_Opacity-5052.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5052.png,Lung_Opacity/masks/Lung_Opacity-5052.png,True,G07906,train,06307307b75729d5ce810327c3590c27,d03f475b0a0f0f2b
Lung_Opacity-5053.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5053.png,Lung_Opacity/masks/Lung_Opacity-5053.png,True,G07907,train,e957eff8153cc6c33eb364ea717e2b9a,d17b521819391b9f
Lung_Opacity-5054.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5054.png,Lung_Opacity/masks/Lung_Opacity-5054.png,True,G07908,train,11cfb36eb973d39e2afb36dd517453ce,897f061a591a597b
Lung_Opacity-5055.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5055.png,Lung_Opacity/masks/Lung_Opacity-5055.png,True,G07909,test,e0c914067aa041a7bb0642b3286dafeb,81fac41e18535f7a
Lung_Opacity-5056.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5056.png,Lung_Opacity/masks/Lung_Opacity-5056.png,True,G07910,test,9d6f077bc28d4ebba7fec92c20378c45,817d07031f0f1f1f
Lung_Opacity-5057.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5057.png,Lung_Opacity/masks/Lung_Opacity-5057.png,True,G07911,val,716d9ba19ab456e271fba5f5ed0c4d2f,853f655a184b7939
Lung_Opacity-5058.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5058.png,Lung_Opacity/masks/Lung_Opacity-5058.png,True,G07912,train,5595d9efc2ecb9381a74826b018d394d,912a461f1e575e5c
Lung_Opacity-5059.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5059.png,Lung_Opacity/masks/Lung_Opacity-5059.png,True,G07913,val,3c0ee174943178d91a053dfe2cd71b25,8b30171b5e5a5aea
Lung_Opacity-506.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-506.png,Lung_Opacity/masks/Lung_Opacity-506.png,True,G07914,train,e1fd20f4b347af5a8239bfbaf9e9489f,cc2c024b0f4f5f5d
Lung_Opacity-5060.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5060.png,Lung_Opacity/masks/Lung_Opacity-5060.png,True,G07915,val,6924dc8bd432ed43a665b32fdbedf380,c03f170a0f4e3e6e
Lung_Opacity-5061.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5061.png,Lung_Opacity/masks/Lung_Opacity-5061.png,True,G07916,train,ac54feb80a23693d7752da0e4658bd79,cc2dd5521f056a3d
Lung_Opacity-5062.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5062.png,Lung_Opacity/masks/Lung_Opacity-5062.png,True,G07917,val,8aa64d01ee599c6d261d01638471b77a,882a875a4f0d7f4e
Lung_Opacity-5063.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5063.png,Lung_Opacity/masks/Lung_Opacity-5063.png,True,G07918,test,d744391dc434883a8c108730ffc0321c,c56a005b6d6c7c5d
Lung_Opacity-5064.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5064.png,Lung_Opacity/masks/Lung_Opacity-5064.png,True,G07919,train,98dc0359be41f00a13c361134e220f86,99698618493f794f
Lung_Opacity-5065.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5065.png,Lung_Opacity/masks/Lung_Opacity-5065.png,True,G07920,train,fa872e250794bfa928375f02ffb0b1e4,9568069e1e5f7859
Lung_Opacity-5066.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5066.png,Lung_Opacity/masks/Lung_Opacity-5066.png,True,G07921,train,e29fcfc8ba5d3ed65ee76910baea56b8,d47f0b5b090e1a5b
Lung_Opacity-5067.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5067.png,Lung_Opacity/masks/Lung_Opacity-5067.png,True,G07922,train,1d3a6691812653554a584f7c35469abe,dd0b935ee8e0a253
Lung_Opacity-5068.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5068.png,Lung_Opacity/masks/Lung_Opacity-5068.png,True,G07923,train,59266282c6111ea9c6a0569647b732f8,d72d7f190d092d2c
Lung_Opacity-5069.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5069.png,Lung_Opacity/masks/Lung_Opacity-5069.png,True,G07924,train,24c9d8b5ca3e84f619c0fdfcdbff89f3,d22d7ac33dc02dd2
Lung_Opacity-507.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-507.png,Lung_Opacity/masks/Lung_Opacity-507.png,True,G07925,train,599f4df211806a0e445ca2b2e8b84ad8,9979071e1a5b1b78
Lung_Opacity-5070.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5070.png,Lung_Opacity/masks/Lung_Opacity-5070.png,True,G07926,train,b09c4937974337f428a7aeac0b51aa70,913b50137b7b7838
Lung_Opacity-5071.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5071.png,Lung_Opacity/masks/Lung_Opacity-5071.png,True,G07927,train,5ba0cb91546f7601fc070eb853661225,817a1c1e5a7b7163
Lung_Opacity-5072.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5072.png,Lung_Opacity/masks/Lung_Opacity-5072.png,True,G07928,train,75cf5d34a626ad38730b465d426bfad1,d42d5a354b5a6a0f
Lung_Opacity-5073.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5073.png,Lung_Opacity/masks/Lung_Opacity-5073.png,True,G07929,val,7cc0fadbc68c7158a9b2191b5e4823e4,d42ee0f38b41a9d5
Lung_Opacity-5074.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5074.png,Lung_Opacity/masks/Lung_Opacity-5074.png,True,G07930,train,ade56c4311af6ba620d4158ca1ee4968,9939471e1e4b1c5e
Lung_Opacity-5075.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5075.png,Lung_Opacity/masks/Lung_Opacity-5075.png,True,G07931,train,272ba46e3a1df2c22e772e42b6dd98c6,953a121b1f1a1e5f
Lung_Opacity-5076.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5076.png,Lung_Opacity/masks/Lung_Opacity-5076.png,True,G07932,test,df7c1068966bbcf32ea6e17c39f35119,9179865e1acd5e38
Lung_Opacity-5077.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5077.png,Lung_Opacity/masks/Lung_Opacity-5077.png,True,G07933,val,ad0c22be0258854a382cdfac1f49772f,8128471f7d495f5a
Lung_Opacity-5078.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5078.png,Lung_Opacity/masks/Lung_Opacity-5078.png,True,G07934,test,3405be66fd8f8d0a156f35d2fa856252,c43cd3591b4a3f1c
Lung_Opacity-5079.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5079.png,Lung_Opacity/masks/Lung_Opacity-5079.png,True,G07935,train,5dae131df0a255bbac43309d69711ad0,c13b13455e1b5d1d
Lung_Opacity-508.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-508.png,Lung_Opacity/masks/Lung_Opacity-508.png,True,G07936,train,a8a9eae051aca9e65c25414ea41f765f,d03f024e1d1c2f3f
Lung_Opacity-5080.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5080.png,Lung_Opacity/masks/Lung_Opacity-5080.png,True,G07937,test,f6e91151ef5b702e1a4050e066f64123,b95dd35b48580b0f
Lung_Opacity-5081.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5081.png,Lung_Opacity/masks/Lung_Opacity-5081.png,True,G07938,train,bd4534935bcacebefa26790ea26b4f2a,91788c391b197f3c
Lung_Opacity-5082.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5082.png,Lung_Opacity/masks/Lung_Opacity-5082.png,True,G07939,train,ec827a2cc5c1064d3e7c6de232330450,d52a061f085d5f5d
Lung_Opacity-5083.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5083.png,Lung_Opacity/masks/Lung_Opacity-5083.png,True,G07940,train,fa3d0f75d9b3e18c8766fd2bc523f37b,817b54105b7f5b16
Lung_Opacity-5084.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5084.png,Lung_Opacity/masks/Lung_Opacity-5084.png,True,G07941,train,c09192aad5eee0ddda7782b35d2fc466,817a5412515b5f5f
Lung_Opacity-5085.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5085.png,Lung_Opacity/masks/Lung_Opacity-5085.png,True,G07942,train,0bf44306fd38fc1b6327e2479a6f293e,c52ad6c0496d6d6d
Lung_Opacity-5086.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5086.png,Lung_Opacity/masks/Lung_Opacity-5086.png,True,G07943,train,8e1e9bac7990a6b84199cf7e357f3283,817f07061f1d0f1f
Lung_Opacity-5087.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5087.png,Lung_Opacity/masks/Lung_Opacity-5087.png,True,G07944,val,dc112692ff9ddd119f2ada89925a2658,d9390b1b296bbaa2
Lung_Opacity-5088.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5088.png,Lung_Opacity/masks/Lung_Opacity-5088.png,True,G07945,val,87fbcb2dcf9a565e71ac87df01db79ab,9879851e1e1d5f19
Lung_Opacity-5089.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5089.png,Lung_Opacity/masks/Lung_Opacity-5089.png,True,G07946,test,742beecb142867e897c548509c43078e,d028c31b296d3ee7
Lung_Opacity-509.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-509.png,Lung_Opacity/masks/Lung_Opacity-509.png,True,G07947,train,fd73a807e35221b61b50abfa73b628d2,d623410e2e64fdf8
Lung_Opacity-5090.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5090.png,Lung_Opacity/masks/Lung_Opacity-5090.png,True,G07948,train,50020fb79e6cf9a5ce98d6310c3be227,917b425b1d091f3d
Lung_Opacity-5091.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5091.png,Lung_Opacity/masks/Lung_Opacity-5091.png,True,G07949,train,6bdd851596cd15e2e09246bc91922bef,cc3dc10f4e4d0f0f
Lung_Opacity-5092.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5092.png,Lung_Opacity/masks/Lung_Opacity-5092.png,True,G07950,train,1051f5ce6a6b8efbd4ead01c8fbf52bd,918d6d986b6d9463
Lung_Opacity-5093.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5093.png,Lung_Opacity/masks/Lung_Opacity-5093.png,True,G07951,train,3ef81b48633a22ef4b6f6d0073bdcaf2,c13d16065f0f5e3a
Lung_Opacity-5094.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5094.png,Lung_Opacity/masks/Lung_Opacity-5094.png,True,G07952,train,d4bdbf513e829b9389c7545dac17159d,d83d62d25a594b59
Lung_Opacity-5095.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5095.png,Lung_Opacity/masks/Lung_Opacity-5095.png,True,G07953,train,1e1d85a69ebe1d199dd10b78877d7574,d93b52d70e054f38
Lung_Opacity-5096.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5096.png,Lung_Opacity/masks/Lung_Opacity-5096.png,True,G07954,train,6c06f34707c5557d9d205a99e3381429,d03d43035b4bce3b
Lung_Opacity-5097.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5097.png,Lung_Opacity/masks/Lung_Opacity-5097.png,True,G07955,val,a9b2acc8ebc595083e2b8af59c3b29a2,d42f53510f0f2a1f
Lung_Opacity-5098.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5098.png,Lung_Opacity/masks/Lung_Opacity-5098.png,True,G07956,test,0e9fe09afc17ac861a1fabd4ebceb8d6,c63d424b0f6e0e2f
Lung_Opacity-5099.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5099.png,Lung_Opacity/masks/Lung_Opacity-5099.png,True,G07957,test,888395573a37f0ebe112ea974ae19599,c46f475a0f4e0e6c
Lung_Opacity-51.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-51.png,Lung_Opacity/masks/Lung_Opacity-51.png,True,G07958,train,cdedd40c7b9a2f6f9f99dad42e90986b,d4bf05515d170b0f
Lung_Opacity-510.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-510.png,Lung_Opacity/masks/Lung_Opacity-510.png,True,G07959,train,6b8112f3b4a44343780adb5e8e5de384,c03d43434f4f4f0f
Lung_Opacity-5100.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5100.png,Lung_Opacity/masks/Lung_Opacity-5100.png,True,G07960,train,62a180cdbae50307b176c138292db138,885d0b1c68fafae2
Lung_Opacity-5101.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5101.png,Lung_Opacity/masks/Lung_Opacity-5101.png,True,G07961,train,1ad573ef6e8ba7ea261f0040d814ce42,dd2a53193d1c9792
Lung_Opacity-5102.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5102.png,Lung_Opacity/masks/Lung_Opacity-5102.png,True,G07962,train,2ad764b95faf8c848b667574a161b0b9,c07f030a0b3e3abf
Lung_Opacity-5103.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5103.png,Lung_Opacity/masks/Lung_Opacity-5103.png,True,G07963,train,cf596b2e6a2fb26e97c8f3db574bb324,95698c1e5b596969
Lung_Opacity-5104.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5104.png,Lung_Opacity/masks/Lung_Opacity-5104.png,True,G07964,train,45dbe29881eb67d3c2b080cda13b9cbd,c42f474a6c6ce4f2
Lung_Opacity-5105.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5105.png,Lung_Opacity/masks/Lung_Opacity-5105.png,True,G07965,train,f51770f66bfc03fbd7047bceda629457,997b461c98eab1f0
Lung_Opacity-5106.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5106.png,Lung_Opacity/masks/Lung_Opacity-5106.png,True,G07966,val,0c1b77275f9cb39456bf418c25b9b32f,996e07584d78ecf0
Lung_Opacity-5107.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5107.png,Lung_Opacity/masks/Lung_Opacity-5107.png,True,G07967,test,ee97d268211dedb2deec71db8738eb23,d42f430b2c2e3f33
Lung_Opacity-5108.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5108.png,Lung_Opacity/masks/Lung_Opacity-5108.png,True,G07968,train,1c4e7ea0ebac2298a76b24237ddd0c86,917fc618195f1f0a
Lung_Opacity-5109.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5109.png,Lung_Opacity/masks/Lung_Opacity-5109.png,True,G07969,val,e44d5ec2a27aaa4fd06862293a4dafa5,c42f834f6e6c68e2
Lung_Opacity-511.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-511.png,Lung_Opacity/masks/Lung_Opacity-511.png,True,G07970,train,fc1ba2fe54f5c3b3d3952307118cbff1,886e475c587c7273
Lung_Opacity-5110.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5110.png,Lung_Opacity/masks/Lung_Opacity-5110.png,True,G07971,train,0969d0bee4ee35d514ff986a34f04869,cc7f434b1f4b050e
Lung_Opacity-5111.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5111.png,Lung_Opacity/masks/Lung_Opacity-5111.png,True,G07972,train,100f2b95dd27a803cebbea334bd22913,cd2e9253490f4f4b
Lung_Opacity-5112.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5112.png,Lung_Opacity/masks/Lung_Opacity-5112.png,True,G07973,train,8be5dc7be99cab4e3da895c4a6566b35,803b1b093bbbaba3
Lung_Opacity-5113.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5113.png,Lung_Opacity/masks/Lung_Opacity-5113.png,True,G07974,train,25224b981999e0a3d7c8af8002f2f623,856e0f5d6f686831
Lung_Opacity-5114.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5114.png,Lung_Opacity/masks/Lung_Opacity-5114.png,True,G07975,train,6604203f11a056c2994ed281863ffd1b,c83d07c61e5c6d59
Lung_Opacity-5115.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5115.png,Lung_Opacity/masks/Lung_Opacity-5115.png,True,G07976,train,1092ecb560c5076a668b6ef20a3e6658,c3343b2d2e3bf1c0
Lung_Opacity-5116.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5116.png,Lung_Opacity/masks/Lung_Opacity-5116.png,True,G07977,val,b5f7b4f1d0a51757f78f8dbb48e94a10,957f061f091d1d4d
Lung_Opacity-5117.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5117.png,Lung_Opacity/masks/Lung_Opacity-5117.png,True,G07978,test,e0cbaed3b00819ae7eecccf1ae5d20db,856d058d5dcd6bc1
Lung_Opacity-5118.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5118.png,Lung_Opacity/masks/Lung_Opacity-5118.png,True,G07979,test,bc29169081dfa73cec89043f58f0c7a0,d52e834d14626afe
Lung_Opacity-5119.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5119.png,Lung_Opacity/masks/Lung_Opacity-5119.png,True,G07980,test,8fe31c22450e7f02d274c2820e3e427e,c435530b0a2f6f2f
Lung_Opacity-512.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-512.png,Lung_Opacity/masks/Lung_Opacity-512.png,True,G07981,train,5205858fc85ff998ce3436766d9a8a8f,9c2fc7971a43115e
Lung_Opacity-5120.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5120.png,Lung_Opacity/masks/Lung_Opacity-5120.png,True,G07982,train,fe8d6264c889abddaa8389a96d3337d1,c87d5a0c2a3db3b1
Lung_Opacity-5121.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5121.png,Lung_Opacity/masks/Lung_Opacity-5121.png,True,G07983,train,91d5aa481571db0db3849b92b25c5ff4,cc3d12424b1f4e6f
Lung_Opacity-5122.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5122.png,Lung_Opacity/masks/Lung_Opacity-5122.png,True,G07984,train,8fbdad73673770b731ab135633b3189a,d02df7120a4d6d5d
Lung_Opacity-5123.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5123.png,Lung_Opacity/masks/Lung_Opacity-5123.png,True,G07985,train,c2d23ea2353bc1b742f1e7c749827b56,917a475c4a5e1c5d
Lung_Opacity-5124.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5124.png,Lung_Opacity/masks/Lung_Opacity-5124.png,True,G07986,train,17dca057761908756f70302a0636527b,9d284f5f58585879
Lung_Opacity-5125.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5125.png,Lung_Opacity/masks/Lung_Opacity-5125.png,True,G07987,train,d2f3b2602c74fdd491a75eaf40b5863c,d42a6f55541a6c6e
Lung_Opacity-5126.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5126.png,Lung_Opacity/masks/Lung_Opacity-5126.png,True,G07988,train,f90da708a8183bc38348cbac9ab6b60a,903f434e1c7938f9
Lung_Opacity-5127.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5127.png,Lung_Opacity/masks/Lung_Opacity-5127.png,True,G07989,train,6954ebed524e11d3f033ed6defdc49d7,d52a074ead171d1e
Lung_Opacity-5128.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5128.png,Lung_Opacity/masks/Lung_Opacity-5128.png,True,G07990,train,02d49a82179c4e829bda2ba0c4656aba,d42f040e0a6f2efd
Lung_Opacity-5129.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5129.png,Lung_Opacity/masks/Lung_Opacity-5129.png,True,G07991,train,1b67c25a87c4f3c7f048c897416f7ace,c83e145a4f4d4d57
Lung_Opacity-513.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-513.png,Lung_Opacity/masks/Lung_Opacity-513.png,True,G07992,val,ed9be0fc3414aadf8836454fa812cbb7,d01a1a1a3a3bbd75
Lung_Opacity-5130.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5130.png,Lung_Opacity/masks/Lung_Opacity-5130.png,True,G07993,val,71ba257fd6520b928d43ea1d251076bd,82721f1117767e72
Lung_Opacity-5131.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5131.png,Lung_Opacity/masks/Lung_Opacity-5131.png,True,G07994,val,3b5ce8ab99dcef3d7d8f1493522fd076,c17f0716961ede06
Lung_Opacity-5132.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5132.png,Lung_Opacity/masks/Lung_Opacity-5132.png,True,G07995,test,b6e61eeae726ff0660f18eedba50c75f,997a041c1a5f5b5b
Lung_Opacity-5133.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5133.png,Lung_Opacity/masks/Lung_Opacity-5133.png,True,G07996,test,ddcaffcfba7808191cfac52859d29abe,d42d4712494f7c7a
Lung_Opacity-5134.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5134.png,Lung_Opacity/masks/Lung_Opacity-5134.png,True,G07997,train,1f7ad6e7d1f718498e57e4cf5d7817a8,c57817430f95686f
Lung_Opacity-5135.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5135.png,Lung_Opacity/masks/Lung_Opacity-5135.png,True,G07998,train,c85d251667769d673eb3cd0b9bc743a9,d42ae25b454a5e6e
Lung_Opacity-5136.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5136.png,Lung_Opacity/masks/Lung_Opacity-5136.png,True,G07999,train,5ecabc18f85fab7ce6458d7c3a84dce5,c03d0b1a3f7a7662
Lung_Opacity-5137.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5137.png,Lung_Opacity/masks/Lung_Opacity-5137.png,True,G08000,train,052de4ab5fd8ef75ab795c0945b812ad,d13f43591e4b391a
Lung_Opacity-5138.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5138.png,Lung_Opacity/masks/Lung_Opacity-5138.png,True,G08001,train,28601743a237c8e2c166b1d50ce74e26,cb6c348b3463d333
Lung_Opacity-5139.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5139.png,Lung_Opacity/masks/Lung_Opacity-5139.png,True,G08002,train,b7b8b8cf993c2d1ca8acfd55f04a583b,817e9454765273c7
Lung_Opacity-514.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-514.png,Lung_Opacity/masks/Lung_Opacity-514.png,True,G08003,train,db38f288091b808f72e16e21700ebaee,da9d25c2474bc12f
Lung_Opacity-5140.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5140.png,Lung_Opacity/masks/Lung_Opacity-5140.png,True,G08004,train,b061f9f300517871fd7aaa39d83c0b32,81780e868e1f9f1f
Lung_Opacity-5141.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5141.png,Lung_Opacity/masks/Lung_Opacity-5141.png,True,G08005,train,251ae4634f20b6ccb7aacb77e0fd9abd,947e831d051f1f0f
Lung_Opacity-5142.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5142.png,Lung_Opacity/masks/Lung_Opacity-5142.png,True,G08006,train,524cae239674b071a65fe09c17f922b3,d4642e2b69e36393
Lung_Opacity-5143.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5143.png,Lung_Opacity/masks/Lung_Opacity-5143.png,True,G08007,train,e0962a5b440d0f19c2bcc0fe14708f5d,913f86161f5c5e58
Lung_Opacity-5144.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5144.png,Lung_Opacity/masks/Lung_Opacity-5144.png,True,G08008,train,588d08548667d327e9cc6e2fd130c906,dc3c4b4b0d1c3c1f
Lung_Opacity-5145.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5145.png,Lung_Opacity/masks/Lung_Opacity-5145.png,True,G08009,train,517e7e04ffc5a81d986cd5bc4c142010,d93a461e383b2fa4
Lung_Opacity-5146.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5146.png,Lung_Opacity/masks/Lung_Opacity-5146.png,True,G08010,val,7710f8d6bda8e1af08e8202067955de9,946fd61e4c1c4c4f
Lung_Opacity-5147.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5147.png,Lung_Opacity/masks/Lung_Opacity-5147.png,True,G08011,val,127c79ca6ab2f6c6e23bd728f65ea99d,d43d430b0f4f4f0e
Lung_Opacity-5148.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5148.png,Lung_Opacity/masks/Lung_Opacity-5148.png,True,G08012,val,df9027e222ad4469a3d3d8cdf4d01b5b,817e571a5d05bc95
Lung_Opacity-5149.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5149.png,Lung_Opacity/masks/Lung_Opacity-5149.png,True,G08013,test,01044e6be2a2fed96069bdefc9f8dbbd,c03d730b086d7d79
Lung_Opacity-515.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-515.png,Lung_Opacity/masks/Lung_Opacity-515.png,True,G08014,train,a21cf8abc6a5e06b4fc8b604d670d13e,d42f07430d5d0f4f
Lung_Opacity-5150.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5150.png,Lung_Opacity/masks/Lung_Opacity-5150.png,True,G08015,test,f2d39b972bea56e864f9b75acd3f3395,9d680e5a796969c5
Lung_Opacity-5151.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5151.png,Lung_Opacity/masks/Lung_Opacity-5151.png,True,G08016,train,6cf0403fa3f761c77be30fa508a9fa4f,917a585a7b3b3a22
Lung_Opacity-5152.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5152.png,Lung_Opacity/masks/Lung_Opacity-5152.png,True,G08017,train,191cc8d790b416d80c3a84f3b71cb8b8,c1295614561f5f5d
Lung_Opacity-5153.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5153.png,Lung_Opacity/masks/Lung_Opacity-5153.png,True,G08018,train,580382fe02343f5176e938bd7f96755c,c13b521a1b393db9
Lung_Opacity-5154.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5154.png,Lung_Opacity/masks/Lung_Opacity-5154.png,True,G08019,test,44634d53f3de33eee5b035a81db08b78,d02d089d5a0aff3d
Lung_Opacity-5155.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5155.png,Lung_Opacity/masks/Lung_Opacity-5155.png,True,G08020,train,2514ddb2357139cf182f810758ed9bf0,9d688f5f48e262e2
Lung_Opacity-5156.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5156.png,Lung_Opacity/masks/Lung_Opacity-5156.png,True,G08021,train,70b5a10796cec5e21f94b8880232e52b,913b525a5f4d1d19
Lung_Opacity-5157.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5157.png,Lung_Opacity/masks/Lung_Opacity-5157.png,True,G08022,train,0ea81c1771a9ec51f08ede97557165c7,d07b524a6b1a2b2f
Lung_Opacity-5158.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5158.png,Lung_Opacity/masks/Lung_Opacity-5158.png,True,G08023,train,4361940d84b3879c5f29f223a52a3783,856c0e5f496b716a
Lung_Opacity-5159.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5159.png,Lung_Opacity/masks/Lung_Opacity-5159.png,True,G08024,train,1b3a5518ec4b08174a038157483e2962,953f2e45c3121b5b
Lung_Opacity-516.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-516.png,Lung_Opacity/masks/Lung_Opacity-516.png,True,G08025,train,6ee3ed6c90a74d16c61c3060ed3722a9,d43fc3515f4e4c0c
Lung_Opacity-5160.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5160.png,Lung_Opacity/masks/Lung_Opacity-5160.png,True,G08026,train,fb3ef813c96020481c8242fe9371d10e,803f041e185f5f5f
Lung_Opacity-5161.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5161.png,Lung_Opacity/masks/Lung_Opacity-5161.png,True,G08027,val,95221de83941f19e3027df118448a81f,9d601f165c5a1b79
Lung_Opacity-5162.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5162.png,Lung_Opacity/masks/Lung_Opacity-5162.png,True,G08028,train,464006e00e1b41c57fe1338faa81ddf6,817be61e98c76e18
Lung_Opacity-5163.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5163.png,Lung_Opacity/masks/Lung_Opacity-5163.png,True,G08029,train,4f106766bd9270ccda18f80454e4be76,81294257555e5e7b
Lung_Opacity-5164.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5164.png,Lung_Opacity/masks/Lung_Opacity-5164.png,True,G08030,train,ace72f820204e0a39e4b6893e3ce5a06,987b431c18fbf8e0
Lung_Opacity-5165.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5165.png,Lung_Opacity/masks/Lung_Opacity-5165.png,True,G08031,test,740acd5aa395831411c1f1c1061936b6,803f531a5d585f4b
Lung_Opacity-5166.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5166.png,Lung_Opacity/masks/Lung_Opacity-5166.png,True,G08032,train,5f9ab0f4fab5c196df4238fbda2bf0ed,816a0e19797e6979
Lung_Opacity-5167.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5167.png,Lung_Opacity/masks/Lung_Opacity-5167.png,True,G08033,train,d0d1c68ad2eea1ec57c161df3ef02997,902fc2d75a433d5c
Lung_Opacity-5168.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5168.png,Lung_Opacity/masks/Lung_Opacity-5168.png,True,G08034,train,f14890b654b10a83a9d20e2897103698,817a063f6de1a786
Lung_Opacity-5169.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5169.png,Lung_Opacity/masks/Lung_Opacity-5169.png,True,G08035,train,0926afa409eb0c6fe32ed3bf31d975f5,c43d471e196db43c
Lung_Opacity-517.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-517.png,Lung_Opacity/masks/Lung_Opacity-517.png,True,G08036,val,80dca2bb01c4cfb2f6df92025c9d2cf4,cc3c1b092dbcbcb1
Lung_Opacity-5170.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5170.png,Lung_Opacity/masks/Lung_Opacity-5170.png,True,G08037,val,c72ce37c67a698ca8c6f9815026f0a68,956817461b57835f
Lung_Opacity-5171.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5171.png,Lung_Opacity/masks/Lung_Opacity-5171.png,True,G08038,test,ac9a30c072b07c451716ebc2d6b3b7bc,d62f9919584b0f4e
Lung_Opacity-5172.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5172.png,Lung_Opacity/masks/Lung_Opacity-5172.png,True,G08039,test,65e26b6ab47440fb52ae24842d09be83,d03fc3131b472d39
Lung_Opacity-5173.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5173.png,Lung_Opacity/masks/Lung_Opacity-5173.png,True,G08040,train,d508de3f0d169a1a20fd0f2e0a08def9,d56802465f191f5f
Lung_Opacity-5174.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5174.png,Lung_Opacity/masks/Lung_Opacity-5174.png,True,G08041,train,c398d892124ebdb3dfb1d10b68b1f073,bf7d0f181e383838
Lung_Opacity-5175.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5175.png,Lung_Opacity/masks/Lung_Opacity-5175.png,True,G08042,train,3056bb9f2f18534574a03d18ccc27fc4,dc2ac1ddc258d2d3
Lung_Opacity-5176.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5176.png,Lung_Opacity/masks/Lung_Opacity-5176.png,True,G08043,train,56c8a9f7313a5b557f8dccfb255a2992,95681d585f5f5858
Lung_Opacity-5177.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5177.png,Lung_Opacity/masks/Lung_Opacity-5177.png,True,G08044,train,474a398987a367ca257e18c420a7e28f,d12a165e387979e1
Lung_Opacity-5178.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5178.png,Lung_Opacity/masks/Lung_Opacity-5178.png,True,G08045,val,52830765513702cc463f0e95d40132bc,813fc61a0d4f3d5c
Lung_Opacity-5179.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5179.png,Lung_Opacity/masks/Lung_Opacity-5179.png,True,G08046,val,bf4e2bf8bdc81dc98fb97be30f255ca7,807b420a1f3b3d3f
Lung_Opacity-518.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-518.png,Lung_Opacity/masks/Lung_Opacity-518.png,True,G08047,train,b762b7beed34ac4ee286cdb6c872f24f,dc0cc3431f49497f
Lung_Opacity-5180.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5180.png,Lung_Opacity/masks/Lung_Opacity-5180.png,True,G08048,test,1fd6d6c65ac68082e064be8572f4f0a4,d33b141c1b1a5bba
Lung_Opacity-5181.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5181.png,Lung_Opacity/masks/Lung_Opacity-5181.png,True,G08049,train,ef8a1b6c9f51fbef7f36aa46a76be764,c03d7b091e1e693b
Lung_Opacity-5182.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5182.png,Lung_Opacity/masks/Lung_Opacity-5182.png,True,G08050,train,0ae852e1c567a50f2a10569669c1ef54,d42f575b4a0b0e1d
Lung_Opacity-5183.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5183.png,Lung_Opacity/masks/Lung_Opacity-5183.png,True,G08051,train,1d7e4308579639a7e942076e7f262146,c05f161a1b3a3de5
Lung_Opacity-5184.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5184.png,Lung_Opacity/masks/Lung_Opacity-5184.png,True,G08052,train,fdcf902385f30a4660d77e19e53310f2,d428470a0b6f2f7e
Lung_Opacity-5185.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5185.png,Lung_Opacity/masks/Lung_Opacity-5185.png,True,G08053,train,92d5662767bebea4181ada9bf16fd5a5,817dc61e385b4b4b
Lung_Opacity-5186.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5186.png,Lung_Opacity/masks/Lung_Opacity-5186.png,True,G08054,train,27f2d8b58326d6e7ead50f414367c36a,d42d43d30b2d6c3e
Lung_Opacity-5187.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5187.png,Lung_Opacity/masks/Lung_Opacity-5187.png,True,G08055,train,9f323083b11ba9f0e3497ada8ec6c73c,8168071e5e5f5a5e
Lung_Opacity-5188.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5188.png,Lung_Opacity/masks/Lung_Opacity-5188.png,True,G08056,val,f670f43d3bf279e039d26c0aa49f8aed,c42fd7416c6c6d0e
Lung_Opacity-5189.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5189.png,Lung_Opacity/masks/Lung_Opacity-5189.png,True,G08057,train,ef481dec30be5dbe6a74f9a36a2220b5,807dcf820e1f8f0f
Lung_Opacity-519.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-519.png,Lung_Opacity/masks/Lung_Opacity-519.png,True,G08058,test,ed3a9261c7ee5714bab1467e5eb91451,8169965e1c876b3b
Lung_Opacity-5190.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5190.png,Lung_Opacity/masks/Lung_Opacity-5190.png,True,G08059,test,a42d8fd1eb5771c985c25a3cb7f7a367,c13b42575b3b2939
Lung_Opacity-5191.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5191.png,Lung_Opacity/masks/Lung_Opacity-5191.png,True,G08060,train,0a8861b2e8385e63b2718751fb08004a,d16a061e1c4f3e7a
Lung_Opacity-5192.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5192.png,Lung_Opacity/masks/Lung_Opacity-5192.png,True,G08061,train,10894b956aa17ed4398782aae8bdec2c,953ea3839f05870f
Lung_Opacity-5193.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5193.png,Lung_Opacity/masks/Lung_Opacity-5193.png,True,G08062,train,9df6825c9353fede206eb2afacf291ea,957e465f185cae30
Lung_Opacity-5194.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5194.png,Lung_Opacity/masks/Lung_Opacity-5194.png,True,G08063,train,b2c1449faa9bd198751c1e5bf79a7b94,913256083b3f3737
Lung_Opacity-5195.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5195.png,Lung_Opacity/masks/Lung_Opacity-5195.png,True,G08064,train,9d1eb486f835ea9fd1619d0477b3438a,c03d4e1e0d6d693d
Lung_Opacity-5196.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5196.png,Lung_Opacity/masks/Lung_Opacity-5196.png,True,G08065,train,c4053a33defc99eb61ab3708eed68fac,807f0f4c0f5c2d5e
Lung_Opacity-5197.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5197.png,Lung_Opacity/masks/Lung_Opacity-5197.png,True,G08066,train,8d3a8ac6f33a7d215da9715735e66797,ca2f474a0e2e2e6e
Lung_Opacity-5198.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5198.png,Lung_Opacity/masks/Lung_Opacity-5198.png,True,G08067,train,2392493eba5939884189fa69d6cb465a,dd2ad4421f574a1d
Lung_Opacity-5199.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5199.png,Lung_Opacity/masks/Lung_Opacity-5199.png,True,G08068,train,fa96ed6a515d1114b4f6367c830288fc,c42d121a597c7c7d
Lung_Opacity-52.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-52.png,Lung_Opacity/masks/Lung_Opacity-52.png,True,G08069,train,dffe2e42535d733f252cd0b851a21090,c56d06521f4c657d
Lung_Opacity-520.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-520.png,Lung_Opacity/masks/Lung_Opacity-520.png,True,G08070,train,c7c64326ac6e2b47f6c764f8e7001e82,997bc61e185b1d19
Lung_Opacity-5200.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5200.png,Lung_Opacity/masks/Lung_Opacity-5200.png,True,G08071,val,84b99334ab3626b6e21ef1d15b10de08,d96e071a4d4d2d6c
Lung_Opacity-5201.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5201.png,Lung_Opacity/masks/Lung_Opacity-5201.png,True,G08072,val,aef36e6cdbfc99f39246436861ca2616,9b781e1a5b5971c4
Lung_Opacity-5202.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5202.png,Lung_Opacity/masks/Lung_Opacity-5202.png,True,G08073,test,e9c69907cc78f755164ed7033e5c018f,dc2ad74b0a0e4f0f
Lung_Opacity-5203.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5203.png,Lung_Opacity/masks/Lung_Opacity-5203.png,True,G08074,val,e83d67f52c4b12b5491b303503ce0a5a,d42e430b2c6e2e7d
Lung_Opacity-5204.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5204.png,Lung_Opacity/masks/Lung_Opacity-5204.png,True,G08075,test,2518d8ce60243ef30c4fc3bb523fd5ca,c07f52125f0b2b3b
Lung_Opacity-5205.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5205.png,Lung_Opacity/masks/Lung_Opacity-5205.png,True,G08076,test,bc01f80e0a4ce5b282ca59d91a696aaf,d13b060e151e3f1f
Lung_Opacity-5206.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5206.png,Lung_Opacity/masks/Lung_Opacity-5206.png,True,G08077,train,5f3e4dee0b1c4855da9e33010681b7b1,d02b5b0a6d29f5b1
Lung_Opacity-5207.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5207.png,Lung_Opacity/masks/Lung_Opacity-5207.png,True,G08078,train,afbfc57ba2503481c850e62e82e53bb7,d02d272f3b21e2f2
Lung_Opacity-5208.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5208.png,Lung_Opacity/masks/Lung_Opacity-5208.png,True,G08079,train,fab0344123481b1b9dbe7ae427ad7ac7,802e0f4b4e6e6e6e
Lung_Opacity-5209.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5209.png,Lung_Opacity/masks/Lung_Opacity-5209.png,True,G08080,train,7abe4c73a055a865e4f2963ed2bd4e80,9c28431e4e6b7aae
Lung_Opacity-521.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-521.png,Lung_Opacity/masks/Lung_Opacity-521.png,True,G08081,train,0ca8d4c5d408275d2d15ccdc622fbcf4,c42f470f4d4c2f4b
Lung_Opacity-5210.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5210.png,Lung_Opacity/masks/Lung_Opacity-5210.png,True,G08082,train,eb78175784f8fa58f7355fa98cbc1ba9,9c7ee2d3c8c1c0da
Lung_Opacity-5211.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5211.png,Lung_Opacity/masks/Lung_Opacity-5211.png,True,G08083,val,3016ac571ff6058276d7e78879157db4,d62f38290f7a2a3a
Lung_Opacity-5212.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5212.png,Lung_Opacity/masks/Lung_Opacity-5212.png,True,G08084,train,1db2b40a06e6e5d0559e7bf7d9db32a1,817f5e196971c1c5
Lung_Opacity-5213.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5213.png,Lung_Opacity/masks/Lung_Opacity-5213.png,True,G08085,test,7a4e47c133c1e7b60abb3275f345a714,d53f96121b585e1a
Lung_Opacity-5214.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5214.png,Lung_Opacity/masks/Lung_Opacity-5214.png,True,G08086,train,5d8f7d3a747294eb6f8a7faf528aa750,9330161f383b3dbc
Lung_Opacity-5215.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5215.png,Lung_Opacity/masks/Lung_Opacity-5215.png,True,G08087,train,e0f1980a0ad69f1a086888e3709b7c1e,d03f4b475f19180f
Lung_Opacity-5216.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5216.png,Lung_Opacity/masks/Lung_Opacity-5216.png,True,G08088,train,72957955cd096f2e61ba7602463037bb,c035db8aa8acadad
Lung_Opacity-5217.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5217.png,Lung_Opacity/masks/Lung_Opacity-5217.png,True,G08089,train,3af86bb604ad9fdc57091bd04b3f1265,d03f4342ad65a5d3
Lung_Opacity-5218.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5218.png,Lung_Opacity/masks/Lung_Opacity-5218.png,True,G08090,train,d8d41b12599e50e96feefc9b04a9e9dc,946e871b0b5d3c6c
Lung_Opacity-5219.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5219.png,Lung_Opacity/masks/Lung_Opacity-5219.png,True,G08091,train,ed48d05eb68219b7c459cb98b2245db6,99281e597be9e1a1
Lung_Opacity-522.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-522.png,Lung_Opacity/masks/Lung_Opacity-522.png,True,G08092,train,12936bcb4eec8f4646c429847f21c68a,c92d07161f4d1d5e
Lung_Opacity-5220.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5220.png,Lung_Opacity/masks/Lung_Opacity-5220.png,True,G08093,val,44501e29dd293daa59d8339692636f67,982cc7571c4b5f1c
Lung_Opacity-5221.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5221.png,Lung_Opacity/masks/Lung_Opacity-5221.png,True,G08094,val,83afb11342143611a3b5edfa120ce929,d47f8713094f0e2e
Lung_Opacity-5222.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5222.png,Lung_Opacity/masks/Lung_Opacity-5222.png,True,G08095,test,19fcd645f3d303a438eb3b1228d69261,907b420b5d0f8f1f
Lung_Opacity-5223.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5223.png,Lung_Opacity/masks/Lung_Opacity-5223.png,True,G08096,test,a6682d4949b043da9b94f15f9fedb3e9,d42d571a3c48785f
Lung_Opacity-5224.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5224.png,Lung_Opacity/masks/Lung_Opacity-5224.png,True,G08097,train,183d8d7174f05b547b001c44b2b4290b,81790f461e131f5f
Lung_Opacity-5225.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5225.png,Lung_Opacity/masks/Lung_Opacity-5225.png,True,G08098,train,1164ea8bdd291114268a676382058ea0,8c4e474b4e6e69e2
Lung_Opacity-5226.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5226.png,Lung_Opacity/masks/Lung_Opacity-5226.png,True,G08099,train,25ea6a0fece425ecec6da3ffe781860f,9a612d151f153f39
Lung_Opacity-5227.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5227.png,Lung_Opacity/masks/Lung_Opacity-5227.png,True,G08100,train,41aa530a5ead04790926110e0cdf078e,81685e196d393dad
Lung_Opacity-5228.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5228.png,Lung_Opacity/masks/Lung_Opacity-5228.png,True,G08101,train,9c1b1ab40039846bf1ab766c4a6472b9,803f175a5d68786b
Lung_Opacity-5229.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5229.png,Lung_Opacity/masks/Lung_Opacity-5229.png,True,G08102,train,2aa81ca53c1be64fc1301402f05fa009,9568a817fa19e90f
Lung_Opacity-523.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-523.png,Lung_Opacity/masks/Lung_Opacity-523.png,True,G08103,train,c83b747a3fb0b14cdeb7fa51dd298453,d52f061e164d5f1c
Lung_Opacity-5230.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5230.png,Lung_Opacity/masks/Lung_Opacity-5230.png,True,G08104,train,861ca6b3bb20ff716cd09edbef681517,817f06061e1f1d3f
Lung_Opacity-5231.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5231.png,Lung_Opacity/masks/Lung_Opacity-5231.png,True,G08105,train,0b4ac2dc40bab542be8eca94db0073ff,817f065a59d958ce
Lung_Opacity-5232.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5232.png,Lung_Opacity/masks/Lung_Opacity-5232.png,True,G08106,train,d0aada529a969849214314a1ef8b04ff,ca3d4bd35b0a1a1e
Lung_Opacity-5233.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5233.png,Lung_Opacity/masks/Lung_Opacity-5233.png,True,G08107,train,505fa1f76ff0190806fbd6c42f86970f,c1b93ac23f37e224
Lung_Opacity-5234.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5234.png,Lung_Opacity/masks/Lung_Opacity-5234.png,True,G08108,train,1b64f6b39acc874c124d3cae895eb2d2,817a141e1c1f7b3b
Lung_Opacity-5235.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5235.png,Lung_Opacity/masks/Lung_Opacity-5235.png,True,G08109,val,33ea18a7fc3845e24e4037dcbd7f85f2,d56e0713593c6873
Lung_Opacity-5236.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5236.png,Lung_Opacity/masks/Lung_Opacity-5236.png,True,G08110,train,bdfc9738ea9c4c5d7e3972822b33251b,d81153524f4b6bab
Lung_Opacity-5237.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5237.png,Lung_Opacity/masks/Lung_Opacity-5237.png,True,G08111,val,8f7f987beef80b07a980faca18c9f9d3,9c2f830b0d1f0f1f
Lung_Opacity-5238.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5238.png,Lung_Opacity/masks/Lung_Opacity-5238.png,True,G08112,val,e924c73eefe6a846f42910ff9331d85f,c03d5d0b1d1d1dbc
Lung_Opacity-5239.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5239.png,Lung_Opacity/masks/Lung_Opacity-5239.png,True,G08113,test,c17603713a9b2a6adfb2533bde636368,95284f4e5e5b2a4e
Lung_Opacity-524.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-524.png,Lung_Opacity/masks/Lung_Opacity-524.png,True,G08114,train,3b6250ec257ab637b57781e0f4aeaa11,956b3c6267869699
Lung_Opacity-5240.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5240.png,Lung_Opacity/masks/Lung_Opacity-5240.png,True,G08115,train,2eba0694a18ff694126fdbb0d3920731,c03f2327b7970b43
Lung_Opacity-5241.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5241.png,Lung_Opacity/masks/Lung_Opacity-5241.png,True,G08116,test,daf726d7374380d19d9384dc2b22c823,c97f16471b1c4d49
Lung_Opacity-5242.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5242.png,Lung_Opacity/masks/Lung_Opacity-5242.png,True,G08117,test,cdde5bce662574182b6ce244d02f9fe7,dd28d31b1d4c2e1e
Lung_Opacity-5243.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5243.png,Lung_Opacity/masks/Lung_Opacity-5243.png,True,G08118,train,b356d047880449970bf2b820563213cd,913b145a7b7be8c0
Lung_Opacity-5244.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5244.png,Lung_Opacity/masks/Lung_Opacity-5244.png,True,G08119,train,6c4adfa1e6479669273accf136d4aef8,917a470512175f5f
Lung_Opacity-5245.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5245.png,Lung_Opacity/masks/Lung_Opacity-5245.png,True,G08120,train,c09a0a03d7c1d93e6c541a20d80c7435,c23d4a0b2d2d2f4f
Lung_Opacity-5246.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5246.png,Lung_Opacity/masks/Lung_Opacity-5246.png,True,G08121,train,e63f6e456cb29862a35cb8dfe3bc5063,817986161e1f3d3d
Lung_Opacity-5247.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5247.png,Lung_Opacity/masks/Lung_Opacity-5247.png,True,G08122,train,304c15b11c30d5773d6a59ddf9c2e84f,cd0e14530f0f4bbd
Lung_Opacity-5248.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5248.png,Lung_Opacity/masks/Lung_Opacity-5248.png,True,G08123,train,2f7fa7bb2cd7ad572e2c7971ff35c4fe,cd0c4a4dede4b6e0
Lung_Opacity-5249.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5249.png,Lung_Opacity/masks/Lung_Opacity-5249.png,True,G08124,train,bd0dcec797e4640cf645132c55e76f88,9d2ac3d7144b1a1f
Lung_Opacity-525.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-525.png,Lung_Opacity/masks/Lung_Opacity-525.png,True,G08125,train,55400bd34d5597b6344c836fdf8b1317,dd3ce24bdd441d18
Lung_Opacity-5250.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5250.png,Lung_Opacity/masks/Lung_Opacity-5250.png,True,G08126,train,4b5cd7f40d78ad2aec76c427b8667b85,d139485f192b1e4f
Lung_Opacity-5251.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5251.png,Lung_Opacity/masks/Lung_Opacity-5251.png,True,G08127,train,b767e512e7c99db63cbb2a26b759fcae,c42d570b1a5b5f1a
Lung_Opacity-5252.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5252.png,Lung_Opacity/masks/Lung_Opacity-5252.png,True,G08128,train,b581ae3789504f89220353333f3aa8ea,c97f841297063b1f
Lung_Opacity-5253.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5253.png,Lung_Opacity/masks/Lung_Opacity-5253.png,True,G08129,train,2bf82ccc9e667629addecb63bf45c78c,d16f9b1b888a8eac
Lung_Opacity-5254.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5254.png,Lung_Opacity/masks/Lung_Opacity-5254.png,True,G08130,val,1582267af72d7c74758449edd3374805,cd26d30d0c4bcb5b
Lung_Opacity-5255.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5255.png,Lung_Opacity/masks/Lung_Opacity-5255.png,True,G08131,test,afbcc0f6463d07daca458332edab7bc3,d02d474e1c6d7c78
Lung_Opacity-5256.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5256.png,Lung_Opacity/masks/Lung_Opacity-5256.png,True,G08132,train,ce4f6e813c33061b0a3f22fd00925b05,997946181f6d39e4
Lung_Opacity-5257.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5257.png,Lung_Opacity/masks/Lung_Opacity-5257.png,True,G08133,val,f62eeae5888cb93dbe13e6f1da4bd776,d42fd74b1d4a491a
Lung_Opacity-5258.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5258.png,Lung_Opacity/masks/Lung_Opacity-5258.png,True,G08134,val,4d66a23323624bf18e25c0d1c8aed9d0,dc2e431a4d4f0c6f
Lung_Opacity-5259.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5259.png,Lung_Opacity/masks/Lung_Opacity-5259.png,True,G08135,test,16bef9c71e64ca053a018387e6382a33,d86947961a4e4f1d
Lung_Opacity-526.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-526.png,Lung_Opacity/masks/Lung_Opacity-526.png,True,G08136,test,815787a83e82985e7717872bec0fc240,897f441ad7431d55
Lung_Opacity-5260.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5260.png,Lung_Opacity/masks/Lung_Opacity-5260.png,True,G08137,train,8d6681a99e5d145e625366d274624bf0,8c2a535f082dbdab
Lung_Opacity-5261.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5261.png,Lung_Opacity/masks/Lung_Opacity-5261.png,True,G08138,train,7f2a30c593f86de786a99ac2e9294cee,c42a530b2c3f3cb7
Lung_Opacity-5262.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5262.png,Lung_Opacity/masks/Lung_Opacity-5262.png,True,G08139,train,d567ebf52904e97a382d58f4d30a04d4,d62b414c4f0b4f4f
Lung_Opacity-5263.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5263.png,Lung_Opacity/masks/Lung_Opacity-5263.png,True,G08140,train,fdf0daa3b292edba080b6e3ec410b8cf,95680e1f5d1d5959
Lung_Opacity-5264.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5264.png,Lung_Opacity/masks/Lung_Opacity-5264.png,True,G08141,train,c6f38c4db28020d4a16483ba55dd1cf3,93781c1c1b5a5b5b
Lung_Opacity-5265.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5265.png,Lung_Opacity/masks/Lung_Opacity-5265.png,True,G08142,train,93558debb114477a558fd36945a4601c,dc2e13193c39697a
Lung_Opacity-5266.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5266.png,Lung_Opacity/masks/Lung_Opacity-5266.png,True,G08143,train,6fc872fb21a8e9429250569aa1057ed7,c43e53434b2d3eac
Lung_Opacity-5267.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5267.png,Lung_Opacity/masks/Lung_Opacity-5267.png,True,G08144,train,15c368c61d934983086d661af11542cb,912946475e1f1b1f
Lung_Opacity-5268.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5268.png,Lung_Opacity/masks/Lung_Opacity-5268.png,True,G08145,train,401ab787a5ab251bb7d3b2d871bd78fc,d17bd7d65458410b
Lung_Opacity-5269.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5269.png,Lung_Opacity/masks/Lung_Opacity-5269.png,True,G08146,train,5b6fd20d940b74b31d062797b9449923,c42f12115e0e6f5f
Lung_Opacity-527.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-527.png,Lung_Opacity/masks/Lung_Opacity-527.png,True,G08147,train,b26a089955e06e3a53c52e8013423303,9b788d145b4a4bcb
Lung_Opacity-5270.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5270.png,Lung_Opacity/masks/Lung_Opacity-5270.png,True,G08148,train,9d33fbc477683e36b96041bf3f5edc04,91780e04171f3f3f
Lung_Opacity-5271.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5271.png,Lung_Opacity/masks/Lung_Opacity-5271.png,True,G08149,train,b7d171593264fde1adcf47d617d9d3e7,d17b561a1b3e1c94
Lung_Opacity-5272.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5272.png,Lung_Opacity/masks/Lung_Opacity-5272.png,True,G08150,train,c04b00a7568cd1af4f75b2f3a6e8171a,d03d430b0f1f3d3c
Lung_Opacity-5273.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5273.png,Lung_Opacity/masks/Lung_Opacity-5273.png,True,G08151,val,d743500ad3bf07f993717caaf23b0882,917f04161b5f5f48
Lung_Opacity-5274.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5274.png,Lung_Opacity/masks/Lung_Opacity-5274.png,True,G08152,val,2c333616e4be7e84269c72470ebba2c1,91784706161e5f5f
Lung_Opacity-5275.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5275.png,Lung_Opacity/masks/Lung_Opacity-5275.png,True,G08153,test,484f0aaec338c2da4e2d1106ad07be82,c429534b4f0f1f2d
Lung_Opacity-5276.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5276.png,Lung_Opacity/masks/Lung_Opacity-5276.png,True,G08154,val,39234e40f770f8ff311b80004eb7eb58,9168078f1f1b1f1b
Lung_Opacity-5277.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5277.png,Lung_Opacity/masks/Lung_Opacity-5277.png,True,G08155,test,c5dd7526ad79e09898bd95eb171b479b,c03d161e5e1e5e5a
Lung_Opacity-5278.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5278.png,Lung_Opacity/masks/Lung_Opacity-5278.png,True,G08156,val,ba980d2f3e88200998d8cf32549a847a,d13b540a5929ebe5
Lung_Opacity-5279.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5279.png,Lung_Opacity/masks/Lung_Opacity-5279.png,True,G08157,test,6434f1242cf2fede510148a734208155,dd2ed20b0e1f292d
Lung_Opacity-528.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-528.png,Lung_Opacity/masks/Lung_Opacity-528.png,True,G08158,train,75f94061b859297bd067df93eacb9a9d,dc19434e1b293dba
Lung_Opacity-5280.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5280.png,Lung_Opacity/masks/Lung_Opacity-5280.png,True,G08159,train,6c4274f93bba795fa942a4d863358d1c,9d7d4253434b4a4f
Lung_Opacity-5281.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5281.png,Lung_Opacity/masks/Lung_Opacity-5281.png,True,G08160,train,109c61cbfefd3285690c70a68e52ed70,dc2d831e5d787868
Lung_Opacity-5282.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5282.png,Lung_Opacity/masks/Lung_Opacity-5282.png,True,G08161,train,0bc396a45be6ac039362f95872f1468d,d42c2bd235d3134f
Lung_Opacity-5283.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5283.png,Lung_Opacity/masks/Lung_Opacity-5283.png,True,G08162,train,d7b0f9a9aa01f1a36cd576c5bb268327,95780f0e1e5e696c
Lung_Opacity-5284.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5284.png,Lung_Opacity/masks/Lung_Opacity-5284.png,True,G08163,train,8232b7e8202b62e80abd91c3802a1e17,981bd65a58535a5d
Lung_Opacity-5285.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5285.png,Lung_Opacity/masks/Lung_Opacity-5285.png,True,G08164,train,b8e2cfe4ceb1a90fa8e09eca48252595,db6bc7d6dad01014
Lung_Opacity-5286.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5286.png,Lung_Opacity/masks/Lung_Opacity-5286.png,True,G08165,train,23dba10f0d9e861972492316f2df313e,857e07425c7d687c
Lung_Opacity-5287.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5287.png,Lung_Opacity/masks/Lung_Opacity-5287.png,True,G08166,train,6bd4f74bdab2ddebdf69a5c4f71ee25d,91784f461a196d6f
Lung_Opacity-5288.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5288.png,Lung_Opacity/masks/Lung_Opacity-5288.png,True,G08167,train,ebaab069df32686b09eb230f5a49179d,dc344a4b0f0b2f2f
Lung_Opacity-5289.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5289.png,Lung_Opacity/masks/Lung_Opacity-5289.png,True,G08168,train,34ade64823da2702b5321dbbfdacf5c6,917f8416125f3b1d
Lung_Opacity-529.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-529.png,Lung_Opacity/masks/Lung_Opacity-529.png,True,G08169,train,58f3e660901a8e850d6dea8a7dc23fac,c87f5b08293c2f9c
Lung_Opacity-5290.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5290.png,Lung_Opacity/masks/Lung_Opacity-5290.png,True,G08170,train,b70407b72821eb20cf52b86f05ab79a8,c02b561b5c093f3f
Lung_Opacity-5291.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5291.png,Lung_Opacity/masks/Lung_Opacity-5291.png,True,G08171,train,a683f8b04ddfbe0c0113ce8f80cdd913,857c0f1f0d1c4d5d
Lung_Opacity-5292.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5292.png,Lung_Opacity/masks/Lung_Opacity-5292.png,True,G08172,val,af8cd1020d52c392737227aba63d9baf,c47dd3026f500b5f
Lung_Opacity-5293.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5293.png,Lung_Opacity/masks/Lung_Opacity-5293.png,True,G08173,val,68501c58dbc4632cbf90d2c2037c8a04,c62753082e372f6e
Lung_Opacity-5294.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5294.png,Lung_Opacity/masks/Lung_Opacity-5294.png,True,G08174,test,95b058109532b097513363be01a14a07,d52f42531b4d4d4d
Lung_Opacity-5295.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5295.png,Lung_Opacity/masks/Lung_Opacity-5295.png,True,G08175,test,18a157860e37770cb20be0adbbbd3399,994a4f191e38d3b9
Lung_Opacity-5296.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5296.png,Lung_Opacity/masks/Lung_Opacity-5296.png,True,G08176,test,6f2d7b89adcdffa15f2ced80de8c8e79,d52f034a7d763131
Lung_Opacity-5297.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5297.png,Lung_Opacity/masks/Lung_Opacity-5297.png,True,G08177,train,a89a71e8b1153e9ffcb0ba80edea0c5e,c92c07171e4b5f59
Lung_Opacity-5298.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5298.png,Lung_Opacity/masks/Lung_Opacity-5298.png,True,G08178,train,c24bc55cbb6dd149d67f35ca9013d0f1,917c0818397d7de9
Lung_Opacity-5299.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5299.png,Lung_Opacity/masks/Lung_Opacity-5299.png,True,G08179,train,e54643f9b0607e0313d9b96011ac199d,c43dc74f4d4c4c1e
Lung_Opacity-53.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-53.png,Lung_Opacity/masks/Lung_Opacity-53.png,True,G08180,train,8c0b7382c03b5dfe4062e12cbcc62871,93608f161e3a7d78
Lung_Opacity-530.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-530.png,Lung_Opacity/masks/Lung_Opacity-530.png,True,G08181,train,9e478a7098d430d7b4bace71bc7b0cb4,b939e5c296c70c1e
Lung_Opacity-5300.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5300.png,Lung_Opacity/masks/Lung_Opacity-5300.png,True,G08182,test,82b1556e7c63f5e75fcaf5a8d0a02a5a,81781f5f3c7a6063
Lung_Opacity-5301.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5301.png,Lung_Opacity/masks/Lung_Opacity-5301.png,True,G08183,train,a09cd874cd5d6214025b2b5c9a27dc94,d52a534b4b2f292d
Lung_Opacity-5302.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5302.png,Lung_Opacity/masks/Lung_Opacity-5302.png,True,G08184,train,9eb95014e65cb9595f3dfe2a2b8f6821,dd38d21f7c0a5c1a
Lung_Opacity-5303.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5303.png,Lung_Opacity/masks/Lung_Opacity-5303.png,True,G08185,train,b0a6d7e52ad353dff431781f09f078e7,842f065f4d5d7d48
Lung_Opacity-5304.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5304.png,Lung_Opacity/masks/Lung_Opacity-5304.png,True,G08186,train,553a6e775d17c659fedd205df435d985,95781c1a5bfbc2c2
Lung_Opacity-5305.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5305.png,Lung_Opacity/masks/Lung_Opacity-5305.png,True,G08187,train,6a9b520e4374b2bec749b2e95eebd19d,d467c14d64666e78
Lung_Opacity-5306.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5306.png,Lung_Opacity/masks/Lung_Opacity-5306.png,True,G08188,train,ed8e7298d69eca8a25b133942eba25e9,d03d031b7d49646f
Lung_Opacity-5307.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5307.png,Lung_Opacity/masks/Lung_Opacity-5307.png,True,G08189,train,43e8dade12510c58d35a23a8fef24f30,846e071d4d6e6a6b
Lung_Opacity-5308.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5308.png,Lung_Opacity/masks/Lung_Opacity-5308.png,True,G08190,train,54b290544610d6591d4e79396685d999,dc35c3cb49492d6c
Lung_Opacity-5309.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5309.png,Lung_Opacity/masks/Lung_Opacity-5309.png,True,G08191,train,e0465d8dec0f44f6b0c226a921d56008,807f8f1e18496d75
Lung_Opacity-531.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-531.png,Lung_Opacity/masks/Lung_Opacity-531.png,True,G08192,val,7a125ca38452764b5726091a9e66229d,dd28534f025f584f
Lung_Opacity-5310.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5310.png,Lung_Opacity/masks/Lung_Opacity-5310.png,True,G08193,val,04a4af656e0e34957953d72f47f81b08,dd6a075b087c6d38
Lung_Opacity-5311.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5311.png,Lung_Opacity/masks/Lung_Opacity-5311.png,True,G08194,test,ddc6fd9f548f56e31573c07d56b3fc70,95691e161e5e1e1e
Lung_Opacity-5312.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5312.png,Lung_Opacity/masks/Lung_Opacity-5312.png,True,G08195,train,a807d6e36552d9002c9f1466fea02c92,903c4746930baf3f
Lung_Opacity-5313.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5313.png,Lung_Opacity/masks/Lung_Opacity-5313.png,True,G08196,val,1e77ef0c4ef72c613dd2918f3372fd9b,89690e1e0a4f4f6f
Lung_Opacity-5314.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5314.png,Lung_Opacity/masks/Lung_Opacity-5314.png,True,G08197,test,0308455fb5d31350f00c94831e605a95,dd2a570f5e411e49
Lung_Opacity-5315.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5315.png,Lung_Opacity/masks/Lung_Opacity-5315.png,True,G08198,test,61ea8c604d6eb1a4802b325ec2b33a11,d47f0b1a0b1a1f1e
Lung_Opacity-5316.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5316.png,Lung_Opacity/masks/Lung_Opacity-5316.png,True,G08199,train,c31e7733ea10fc88b550ed3fe7b49512,dd28d28396c0ab9f
Lung_Opacity-5317.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5317.png,Lung_Opacity/masks/Lung_Opacity-5317.png,True,G08200,train,411c48dc78d7718a51645b4ecf75369b,d52b514b2e2e2a3b
Lung_Opacity-5318.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5318.png,Lung_Opacity/masks/Lung_Opacity-5318.png,True,G08201,val,8f3f780d268e9f0d11b2a892312ac1a9,ce6d0e793133323a
Lung_Opacity-5319.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5319.png,Lung_Opacity/masks/Lung_Opacity-5319.png,True,G08202,train,62272b8445d712a9ff1cd8d746e119f9,c138124b495f5b5f
Lung_Opacity-532.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-532.png,Lung_Opacity/masks/Lung_Opacity-532.png,True,G08203,train,fb95fb182b4925df4f92e4577b1ed9e6,dc1d4b090e1d1e3f
Lung_Opacity-5320.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5320.png,Lung_Opacity/masks/Lung_Opacity-5320.png,True,G08204,train,830ed14037998ff57307d21a99449d7e,902bc71a4e0b3f2f
Lung_Opacity-5321.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5321.png,Lung_Opacity/masks/Lung_Opacity-5321.png,True,G08205,train,d4a172225d85a7db8f34ed343860608c,817f025e7b6b6524
Lung_Opacity-5322.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5322.png,Lung_Opacity/masks/Lung_Opacity-5322.png,True,G08206,train,ce7f2112031b23d223da0459456e2256,9b608e1ff81b780b
Lung_Opacity-5323.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5323.png,Lung_Opacity/masks/Lung_Opacity-5323.png,True,G08207,train,51dfcd2fcd601fd1018a993a97e8c091,d920561b2c6f393d
Lung_Opacity-5324.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5324.png,Lung_Opacity/masks/Lung_Opacity-5324.png,True,G08208,train,f54fdfe72648816a7cace6fea1b05a57,cd7e534f0c293c0d
Lung_Opacity-5325.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5325.png,Lung_Opacity/masks/Lung_Opacity-5325.png,True,G08209,train,f1a6edc1b9b88e38759cf0de48e3c7f3,d52ba05119175f5e
Lung_Opacity-5326.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5326.png,Lung_Opacity/masks/Lung_Opacity-5326.png,True,G08210,train,7cb3864400bf58e7048446990d74ef90,912bee10191b3f1f
Lung_Opacity-5327.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5327.png,Lung_Opacity/masks/Lung_Opacity-5327.png,True,G08211,train,e6143aec802499ba37fdeb70620a0962,dc2ed34b0f0e0e2e
Lung_Opacity-5328.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5328.png,Lung_Opacity/masks/Lung_Opacity-5328.png,True,G08212,train,5d1bcb7919db4d82ee0b379e31840058,d0295a16df4a5e4e
Lung_Opacity-5329.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5329.png,Lung_Opacity/masks/Lung_Opacity-5329.png,True,G08213,val,6806df794925327cf28d33d6fbbc0c7e,81689d5d5c5c6dc6
Lung_Opacity-533.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-533.png,Lung_Opacity/masks/Lung_Opacity-533.png,True,G08214,val,c8b2df1aff83c713022bd44403d7dc63,91280f1f795d39d1
Lung_Opacity-5330.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5330.png,Lung_Opacity/masks/Lung_Opacity-5330.png,True,G08215,val,9ef01c1e73eee09c42e751c6ec754a30,8939121b193d3dbd
Lung_Opacity-5331.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5331.png,Lung_Opacity/masks/Lung_Opacity-5331.png,True,G08216,test,c1ae20856898c1b2f0f53b180434c286,997f051c184b5b5b
Lung_Opacity-5332.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5332.png,Lung_Opacity/masks/Lung_Opacity-5332.png,True,G08217,test,6765394ace9b721250004cbfb72c1663,947d4f561d4d0d1c
Lung_Opacity-5333.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5333.png,Lung_Opacity/masks/Lung_Opacity-5333.png,True,G08218,train,f55735b7b7c6674717da0940a7bfdfc2,987d86165aea9ec8
Lung_Opacity-5334.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5334.png,Lung_Opacity/masks/Lung_Opacity-5334.png,True,G08219,train,69b1ea0edd1b6e1a5ac52db0a0b9fd4a,9f613c9e3c18683b
Lung_Opacity-5335.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5335.png,Lung_Opacity/masks/Lung_Opacity-5335.png,True,G08220,train,873cddc0685099f4def21f7e0911a609,95205f5d395b3938
Lung_Opacity-5336.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5336.png,Lung_Opacity/masks/Lung_Opacity-5336.png,True,G08221,train,b9ad9261b645d3d64263eda295d0ad26,890c160b7f797978
Lung_Opacity-5337.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5337.png,Lung_Opacity/masks/Lung_Opacity-5337.png,True,G08222,train,b7fa8c26cd1d71b3d53d92d03da7b5bf,c42dc3cb0d4c4f4f
Lung_Opacity-5338.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5338.png,Lung_Opacity/masks/Lung_Opacity-5338.png,True,G08223,train,d7def3c291ee0778727fb1b3db2b7d5e,c82e82cb0d5e0f5f
Lung_Opacity-5339.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5339.png,Lung_Opacity/masks/Lung_Opacity-5339.png,True,G08224,train,008e67da7fe49a40244f3e18ccb398af,89699e1a195d7969
Lung_Opacity-534.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-534.png,Lung_Opacity/masks/Lung_Opacity-534.png,True,G08225,train,bc2acc4bcab0e41247231f0e67762e03,dd2a030f0d5d5c57
Lung_Opacity-5340.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5340.png,Lung_Opacity/masks/Lung_Opacity-5340.png,True,G08226,train,e42261bf95ce3c9b4145c582f156c353,997b871a4b90d8e9
Lung_Opacity-5341.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5341.png,Lung_Opacity/masks/Lung_Opacity-5341.png,True,G08227,train,e861520139a1d500cb5045d9eac07337,cd3c031b5b4cfab0
Lung_Opacity-5342.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5342.png,Lung_Opacity/masks/Lung_Opacity-5342.png,True,G08228,train,c0da9e496e5be153863735b67db6d3db,dd395a1c1a2626b7
Lung_Opacity-5343.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5343.png,Lung_Opacity/masks/Lung_Opacity-5343.png,True,G08229,train,7a3303c8ee81eaca64c9572b0d65d2a1,842a434b4b437bfb
Lung_Opacity-5344.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5344.png,Lung_Opacity/masks/Lung_Opacity-5344.png,True,G08230,train,eac3d5a2ca312082a97a9b3846673dc6,d03b121a1b1bbebc
Lung_Opacity-5345.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5345.png,Lung_Opacity/masks/Lung_Opacity-5345.png,True,G08231,train,593e3781cfa92a4dcc81a8645dd2b169,c03bd61f165a5a5a
Lung_Opacity-5346.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5346.png,Lung_Opacity/masks/Lung_Opacity-5346.png,True,G08232,val,de8cb2d721139a624669b2b09ab5e132,c52d165e4a4dfc68
Lung_Opacity-5347.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5347.png,Lung_Opacity/masks/Lung_Opacity-5347.png,True,G08233,val,a45a0e366946a0321fb91e34ebcaa5d7,d5242e0fae4e6c4e
Lung_Opacity-5348.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5348.png,Lung_Opacity/masks/Lung_Opacity-5348.png,True,G08234,val,e9f4de2c704bdf92ed841c763dc4aebe,897f82d61a5e085f
Lung_Opacity-5349.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5349.png,Lung_Opacity/masks/Lung_Opacity-5349.png,True,G08235,train,16d7f622fd13fc8077cc2ce9669b3827,887f8617092d3d6d
Lung_Opacity-535.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-535.png,Lung_Opacity/masks/Lung_Opacity-535.png,True,G08236,test,7f3e8d3d61070cb01e030b5209b2147d,dd29421e5a693cec
Lung_Opacity-5350.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5350.png,Lung_Opacity/masks/Lung_Opacity-5350.png,True,G08237,test,5ffe75f162c0799fddd5d989592e678a,c53a3d2d1c2d2d39
Lung_Opacity-5351.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5351.png,Lung_Opacity/masks/Lung_Opacity-5351.png,True,G08238,train,e88879471ad727fc86f0782915f3a711,89680c565b4f4f67
Lung_Opacity-5352.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5352.png,Lung_Opacity/masks/Lung_Opacity-5352.png,True,G08239,test,cb4b988734ed8efab461fc134ffd88f1,d66a814b6a666f68
Lung_Opacity-5353.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5353.png,Lung_Opacity/masks/Lung_Opacity-5353.png,True,G08240,test,20ee517ea52acabbfe33648371f41fc3,d42e12534c6cedf2
Lung_Opacity-5354.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5354.png,Lung_Opacity/masks/Lung_Opacity-5354.png,True,G08241,train,e423386dfb9041e28fb76830e5ea1d82,c12f065a4d496fe6
Lung_Opacity-5355.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5355.png,Lung_Opacity/masks/Lung_Opacity-5355.png,True,G08242,train,2804a5a614ec3b49323a7bc76b8ed12c,d029485b5b1b3b3b
Lung_Opacity-5356.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5356.png,Lung_Opacity/masks/Lung_Opacity-5356.png,True,G08243,train,1a3ca7a808605fe20856ddbb180a35a4,d03bd2564a4b5b59
Lung_Opacity-5357.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5357.png,Lung_Opacity/masks/Lung_Opacity-5357.png,True,G08244,train,56dbc3f838d994537a13b69b22a037ed,903f2d071e1f1f1c
Lung_Opacity-5358.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5358.png,Lung_Opacity/masks/Lung_Opacity-5358.png,True,G08245,train,af1239da67b527b65e71cb0e731fbd9b,dd2a56071f404b7d
Lung_Opacity-5359.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5359.png,Lung_Opacity/masks/Lung_Opacity-5359.png,True,G08246,train,d19edeffb4de2535dabb31540497f9fc,d92bc59695131756
Lung_Opacity-536.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-536.png,Lung_Opacity/masks/Lung_Opacity-536.png,True,G08247,train,d0166840251a7b60e60ee4836bdfce0c,c81d0a4f4fa8fda1
Lung_Opacity-5360.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5360.png,Lung_Opacity/masks/Lung_Opacity-5360.png,True,G08248,val,35e4dde17f71a76e1bc5c4bc24a3fa2d,d07a53195b0b1a9f
Lung_Opacity-5361.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5361.png,Lung_Opacity/masks/Lung_Opacity-5361.png,True,G08249,train,ea35b9ebc65e9b884ec37a12c8ae3f82,813f365a696de845
Lung_Opacity-5362.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5362.png,Lung_Opacity/masks/Lung_Opacity-5362.png,True,G08250,val,dd02c041616f85507ce8c441ce9d3c8a,c93d174e1c585b4d
Lung_Opacity-5363.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5363.png,Lung_Opacity/masks/Lung_Opacity-5363.png,True,G08251,train,78e2c06a5487f54152045778b356dce1,817996061e59797d
Lung_Opacity-5364.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5364.png,Lung_Opacity/masks/Lung_Opacity-5364.png,True,G08252,train,9c44a69a93ae403d824f09e89eaf2984,c52e06161a5b5d7e
Lung_Opacity-5365.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5365.png,Lung_Opacity/masks/Lung_Opacity-5365.png,True,G08253,train,b94eacb3dd7753ccbb28f1e448369f05,813bd6121939795f
Lung_Opacity-5366.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5366.png,Lung_Opacity/masks/Lung_Opacity-5366.png,True,G08254,train,cd63f8aa41b3282344baa04541b99035,917f06063a113f3f
Lung_Opacity-5367.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5367.png,Lung_Opacity/masks/Lung_Opacity-5367.png,True,G08255,val,a4f8ef13745c124f25f5b70845761f69,d03fd5027d0aeb43
Lung_Opacity-5368.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5368.png,Lung_Opacity/masks/Lung_Opacity-5368.png,True,G08256,test,3d120fb5c02e9bd42362696c9767a988,d93d820717471f1d
Lung_Opacity-5369.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5369.png,Lung_Opacity/masks/Lung_Opacity-5369.png,True,G08257,test,3bce09093911a6843995b1e575df76e0,d93f851e950f1b0c
Lung_Opacity-537.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-537.png,Lung_Opacity/masks/Lung_Opacity-537.png,True,G08258,train,e58e87ec8244fb7903e770efdc01b73a,81681e1a5f5b7a59
Lung_Opacity-5370.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5370.png,Lung_Opacity/masks/Lung_Opacity-5370.png,True,G08259,train,2ecafb3586d60d10847b8436a99ecbc8,91704f5e5a39383d
Lung_Opacity-5371.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5371.png,Lung_Opacity/masks/Lung_Opacity-5371.png,True,G08260,train,6741e94c7ea6cf9aa7a902a9bcf8423a,dc6d430b4f6a2e2a
Lung_Opacity-5372.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5372.png,Lung_Opacity/masks/Lung_Opacity-5372.png,True,G08261,train,1f3791ff703595f717848edfcec9645c,9f6d4e0f29f57020
Lung_Opacity-5373.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5373.png,Lung_Opacity/masks/Lung_Opacity-5373.png,True,G08262,test,238c1bb0cdc884eab121ad2c5884e8cf,c9791506965f7c1c
Lung_Opacity-5374.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5374.png,Lung_Opacity/masks/Lung_Opacity-5374.png,True,G08263,train,96fa54dd1193a80a6a7ce2276cd6764f,c23f35d831d22fc2
Lung_Opacity-5375.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5375.png,Lung_Opacity/masks/Lung_Opacity-5375.png,True,G08264,train,e32b35f0f3bf109250c2f0a9a366c229,c03fd34b1e1a4d0f
Lung_Opacity-5376.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5376.png,Lung_Opacity/masks/Lung_Opacity-5376.png,True,G08265,val,885dbd6b2afc431e1bfd532994529516,817def061416be1b
Lung_Opacity-5377.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5377.png,Lung_Opacity/masks/Lung_Opacity-5377.png,True,G08266,train,30bc45846275d5efa80ea932edb08025,807fc6de58413b5a
Lung_Opacity-5378.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5378.png,Lung_Opacity/masks/Lung_Opacity-5378.png,True,G08267,train,93cc6c3f63cc8fb899be5a00cec50c3e,d56c2e027af570b1
Lung_Opacity-5379.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5379.png,Lung_Opacity/masks/Lung_Opacity-5379.png,True,G08268,train,7b7fad0f79d445946253d034dba69007,9c3d421f0a6d5a2f
Lung_Opacity-538.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-538.png,Lung_Opacity/masks/Lung_Opacity-538.png,True,G08269,train,00ffe7d507d6a5e09fe49eccad51dd98,c93a160d9e3ab8ea
Lung_Opacity-5380.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5380.png,Lung_Opacity/masks/Lung_Opacity-5380.png,True,G08270,train,fd09acfda7dfe50c164013a8371cc5d2,c62c2b4b77f23283
Lung_Opacity-5381.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5381.png,Lung_Opacity/masks/Lung_Opacity-5381.png,True,G08271,test,c1cc0db2a748163b737b1cb6dd8546f5,c56f47434d682b2b
Lung_Opacity-5382.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5382.png,Lung_Opacity/masks/Lung_Opacity-5382.png,True,G08272,train,4d41341cc64bb6aebe75908f8702b37a,99384e1f386d6939
Lung_Opacity-5383.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5383.png,Lung_Opacity/masks/Lung_Opacity-5383.png,True,G08273,train,4beaa21f3987f4e7cedbb0d211a7ed0c,c939169f59b9e1a0
Lung_Opacity-5384.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5384.png,Lung_Opacity/masks/Lung_Opacity-5384.png,True,G08274,train,52c14a6c76ea8a377e41f6ce7b717bcb,d42fc71b1c4f0b4c
Lung_Opacity-5385.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5385.png,Lung_Opacity/masks/Lung_Opacity-5385.png,True,G08275,val,78f93ac7e7579738f7800552ee46b679,901b5a1a1abbbeb8
Lung_Opacity-5386.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5386.png,Lung_Opacity/masks/Lung_Opacity-5386.png,True,G08276,val,01ae80743ddb6f332ac9dc8379cc3ec6,997b061e1e1b29ba
Lung_Opacity-5387.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5387.png,Lung_Opacity/masks/Lung_Opacity-5387.png,True,G08277,test,7717f178509c1153e89529c510a56cb1,dd2d8612174f1e17
Lung_Opacity-5388.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5388.png,Lung_Opacity/masks/Lung_Opacity-5388.png,True,G08278,test,7fbb3326d31717ad91b896f0516b3e89,c52d3ab2521b785b
Lung_Opacity-5389.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5389.png,Lung_Opacity/masks/Lung_Opacity-5389.png,True,G08279,train,34799b5936c077fc82e40a28ec850727,917b07161e1b1f3a
Lung_Opacity-539.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-539.png,Lung_Opacity/masks/Lung_Opacity-539.png,True,G08280,train,1e5f8a63a30eec82bf74d136cb7f0166,c43f0a2ca2afe3a3
Lung_Opacity-5390.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5390.png,Lung_Opacity/masks/Lung_Opacity-5390.png,True,G08281,train,78c16c14a99873bad55bcf9aeaa7737d,946f434acb4eaf44
Lung_Opacity-5391.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5391.png,Lung_Opacity/masks/Lung_Opacity-5391.png,True,G08282,train,a15fce51e8d2a47d02ae12db8170f34b,cd2c87171d5c6875
Lung_Opacity-5392.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5392.png,Lung_Opacity/masks/Lung_Opacity-5392.png,True,G08283,train,5841d74be91f07c72127e4ddca2a5bc9,81ba2f4f1c1d194f
Lung_Opacity-5393.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5393.png,Lung_Opacity/masks/Lung_Opacity-5393.png,True,G08284,val,dae7600695c25c6fc6d702058132e6ee,d42c12cf4e1cff81
Lung_Opacity-5394.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5394.png,Lung_Opacity/masks/Lung_Opacity-5394.png,True,G08285,train,d32d25581fab3aeb65ae58e08ed468c0,9b68141e1d171f5d
Lung_Opacity-5395.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5395.png,Lung_Opacity/masks/Lung_Opacity-5395.png,True,G08286,train,83a362e77bd68fc0b5bea303427abdd9,d52e6e025c496d6d
Lung_Opacity-5396.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5396.png,Lung_Opacity/masks/Lung_Opacity-5396.png,True,G08287,train,e4f354c8526ea054bbcb01966d40711e,907fc716424f7949
Lung_Opacity-5397.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5397.png,Lung_Opacity/masks/Lung_Opacity-5397.png,True,G08288,train,24d452c2f218d8651279c7a9ee9f057e,993b071a195b1afa
Lung_Opacity-5398.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5398.png,Lung_Opacity/masks/Lung_Opacity-5398.png,True,G08289,train,02e8c5e7e7181feb53fd0825c84a1ccf,c13b5e8819d93d95
Lung_Opacity-5399.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5399.png,Lung_Opacity/masks/Lung_Opacity-5399.png,True,G08290,train,bf591cbd87f7ed6307e2889f743bc60e,d42b4f1d4d283c2f
Lung_Opacity-54.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-54.png,Lung_Opacity/masks/Lung_Opacity-54.png,True,G08291,train,6efa74923ab6ee8c7ae344fc35bf1a27,882ad51f575a1b4e
Lung_Opacity-540.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-540.png,Lung_Opacity/masks/Lung_Opacity-540.png,True,G08292,train,614deb430328ee638eaad3e3d5e0c50d,c52f074f0d4968f9
Lung_Opacity-5400.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5400.png,Lung_Opacity/masks/Lung_Opacity-5400.png,True,G08293,train,da17d66da6d1decde441a063a062f38a,dc3d53434b4b4b0d
Lung_Opacity-5401.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5401.png,Lung_Opacity/masks/Lung_Opacity-5401.png,True,G08294,train,66bfc70cbcc45b00a0b1bb166aeefae5,c12d16534b2f3d3c
Lung_Opacity-5402.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5402.png,Lung_Opacity/masks/Lung_Opacity-5402.png,True,G08295,val,c6b61211f51dd72b293a470f5e2f3766,dc2dd14b4d0d2f0b
Lung_Opacity-5403.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5403.png,Lung_Opacity/masks/Lung_Opacity-5403.png,True,G08296,val,18c7a226932c25dacdef06dd37d6a567,da3d4b4c0b2d2c3d
Lung_Opacity-5404.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5404.png,Lung_Opacity/masks/Lung_Opacity-5404.png,True,G08297,test,28d66e6b4a39216ee15139e96dd66c62,d52e071d0c4f4ecb
Lung_Opacity-5405.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5405.png,Lung_Opacity/masks/Lung_Opacity-5405.png,True,G08298,test,cf3f0fabc4a88641411d1cb710df5265,c96f141a1d1e1a5f
Lung_Opacity-5406.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5406.png,Lung_Opacity/masks/Lung_Opacity-5406.png,True,G08299,train,53548697ef22c99079d0beeebb5952fa,dd2b47871e0d4e0e
Lung_Opacity-5407.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5407.png,Lung_Opacity/masks/Lung_Opacity-5407.png,True,G08300,train,d62d9a1c1e5ddd66ee2ef0ce51f67fdc,d42b025b7a7c7c70
Lung_Opacity-5408.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5408.png,Lung_Opacity/masks/Lung_Opacity-5408.png,True,G08301,train,f64cf3bbff6ed5f0cf930a5f0ebc4dff,d43d431b09383df5
Lung_Opacity-5409.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5409.png,Lung_Opacity/masks/Lung_Opacity-5409.png,True,G08302,train,7f6bda4ce58986f0e5600fd83442ff11,827b85051f1f1b3b
Lung_Opacity-541.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-541.png,Lung_Opacity/masks/Lung_Opacity-541.png,True,G08303,train,a38291718f29b754bcab4bb63a2f8c55,dc3d421b084d3fe9
Lung_Opacity-5410.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5410.png,Lung_Opacity/masks/Lung_Opacity-5410.png,True,G08304,train,a86acf27ac9dad00b2b7b24413174775,aa2fc34b2f4a0f0e
Lung_Opacity-5411.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5411.png,Lung_Opacity/masks/Lung_Opacity-5411.png,True,G08305,train,69e749142402a87606d7d117c86001b7,c02d02926f4fda3f
Lung_Opacity-5412.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5412.png,Lung_Opacity/masks/Lung_Opacity-5412.png,True,G08306,train,fc26153678c2281f57d896d24f55a818,d42a530b0e6e3f36
Lung_Opacity-5413.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5413.png,Lung_Opacity/masks/Lung_Opacity-5413.png,True,G08307,train,dfbec3c04039ed5a3ae86f688c3bd59e,d12b541a1929bdbd
Lung_Opacity-5414.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5414.png,Lung_Opacity/masks/Lung_Opacity-5414.png,True,G08308,train,48f9fb5b0c7b728874fdbc8091c08f67,90bdeb4638f3c704
Lung_Opacity-5415.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5415.png,Lung_Opacity/masks/Lung_Opacity-5415.png,True,G08309,train,2ffc657144bd86b87d59c210bfda0990,817a68195b1b5b5b
Lung_Opacity-5416.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5416.png,Lung_Opacity/masks/Lung_Opacity-5416.png,True,G08310,train,c89920cb345cd4691e3bdbd9adbaf1a1,d568c30b0f0f3c3d
Lung_Opacity-5417.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5417.png,Lung_Opacity/masks/Lung_Opacity-5417.png,True,G08311,train,ebf88102f09e4cdfa0de63f15bf91be6,d03d3ad22de4236d
Lung_Opacity-5418.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5418.png,Lung_Opacity/masks/Lung_Opacity-5418.png,True,G08312,train,7db3114f16c646e0dca044dec164e006,9969a6075e12c74f
Lung_Opacity-5419.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5419.png,Lung_Opacity/masks/Lung_Opacity-5419.png,True,G08313,val,6696721eee993fcd0568498d27058a78,903eaf01cb0b1f2f
Lung_Opacity-542.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-542.png,Lung_Opacity/masks/Lung_Opacity-542.png,True,G08314,val,68a18dc40155f0608c6dc71d5e84ca52,9979421e1c7b38bc
Lung_Opacity-5420.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5420.png,Lung_Opacity/masks/Lung_Opacity-5420.png,True,G08315,val,673a95922eb6c7ef3123bbb5e0df7ea6,8968071a7b793973
Lung_Opacity-5421.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5421.png,Lung_Opacity/masks/Lung_Opacity-5421.png,True,G08316,test,2cc1104244461cc6238a4a0079989338,d43e45c3d15b5b50
Lung_Opacity-5422.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5422.png,Lung_Opacity/masks/Lung_Opacity-5422.png,True,G08317,test,f3084ea8d11de4b6ea2c0de4595f4318,dc2a434b0d8e1ebd
Lung_Opacity-5423.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5423.png,Lung_Opacity/masks/Lung_Opacity-5423.png,True,G08318,train,74280f55e01b8f150c9479b5685864ea,85680626af5fb43b
Lung_Opacity-5424.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5424.png,Lung_Opacity/masks/Lung_Opacity-5424.png,True,G08319,test,7ea2e1b45b7a4d80a25c82c0921b7705,c93f164e1c5c1d59
Lung_Opacity-5425.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5425.png,Lung_Opacity/masks/Lung_Opacity-5425.png,True,G08320,train,e0957e73365d63670c4aa0df953f9b5b,817e0e4c1c796bc7
Lung_Opacity-5426.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5426.png,Lung_Opacity/masks/Lung_Opacity-5426.png,True,G08321,train,e5d9b3ad3e3e77bde46ecde85a4e7546,c47c0a0f6a73b1b3
Lung_Opacity-5427.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5427.png,Lung_Opacity/masks/Lung_Opacity-5427.png,True,G08322,train,55e18a36facdc6542d3ad8309e323fcb,d12942037f095f7b
Lung_Opacity-5428.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5428.png,Lung_Opacity/masks/Lung_Opacity-5428.png,True,G08323,train,ed9bbc771e9bacac76773cb4c0d9d316,c22f080cbdf9b13b
Lung_Opacity-5429.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5429.png,Lung_Opacity/masks/Lung_Opacity-5429.png,True,G08324,train,b90e46156c740f96451aff67b2c8fff8,811b1c5a7b1b5a7a
Lung_Opacity-543.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-543.png,Lung_Opacity/masks/Lung_Opacity-543.png,True,G08325,train,a77ec2fd382155d8bef7dd147fa5360d,c03d5d4b4a2e2e4f
Lung_Opacity-5430.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5430.png,Lung_Opacity/masks/Lung_Opacity-5430.png,True,G08326,val,99537be0a658c4a4aa70f4ab936c73a3,c16e422a7535a6b7
Lung_Opacity-5431.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5431.png,Lung_Opacity/masks/Lung_Opacity-5431.png,True,G08327,train,fda5d036ce5bf2ca226fe2994c228d21,d52ad21a1f4b0f1d
Lung_Opacity-5432.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5432.png,Lung_Opacity/masks/Lung_Opacity-5432.png,True,G08328,train,f90dfb694677a13361212ce93d8df653,857a60525b5b5a5b
Lung_Opacity-5433.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5433.png,Lung_Opacity/masks/Lung_Opacity-5433.png,True,G08329,train,46ec57aa5e52a413129e13760217c605,c43f03471d4d0f4f
Lung_Opacity-5434.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5434.png,Lung_Opacity/masks/Lung_Opacity-5434.png,True,G08330,train,a646135226d3c526a6c88bf461b53e8b,c42e570b2a2e2f79
Lung_Opacity-5435.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5435.png,Lung_Opacity/masks/Lung_Opacity-5435.png,True,G08331,train,fafe63952962b29c843458e60dc05227,913ae01e395b4b4f
Lung_Opacity-5436.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5436.png,Lung_Opacity/masks/Lung_Opacity-5436.png,True,G08332,train,1cc7f258c0e4fb8c1c9835d0ac4b4fbf,913bd64a51333337
Lung_Opacity-5437.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5437.png,Lung_Opacity/masks/Lung_Opacity-5437.png,True,G08333,train,e2b38ce0b444cdd8d5ffb50ed724bcd6,817a74e3a79cd8c1
Lung_Opacity-5438.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5438.png,Lung_Opacity/masks/Lung_Opacity-5438.png,True,G08334,val,ecb7fd5946f7506f6510a2d4302fc550,917a06161e5b3b5b
Lung_Opacity-5439.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5439.png,Lung_Opacity/masks/Lung_Opacity-5439.png,True,G08335,val,44a43415207dff260d1427037dc7b240,893957525b5d590e
Lung_Opacity-544.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-544.png,Lung_Opacity/masks/Lung_Opacity-544.png,True,G08336,test,0091214b43763737b61590f2ec0f4f82,ce0dd30f0f1d9692
Lung_Opacity-5440.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5440.png,Lung_Opacity/masks/Lung_Opacity-5440.png,True,G08337,test,d8cfa2a7f724a01f1d8b1af4b1659d75,d02b92121e5e1f5f
Lung_Opacity-5441.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5441.png,Lung_Opacity/masks/Lung_Opacity-5441.png,True,G08338,test,79797dfc677e8b2652347146e7646ffe,d40d1b0b0d3c78fd
Lung_Opacity-5442.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5442.png,Lung_Opacity/masks/Lung_Opacity-5442.png,True,G08339,train,73b816109657d6dbcde96f88bd47ad2f,dc29d30f6c0afd30
Lung_Opacity-5443.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5443.png,Lung_Opacity/masks/Lung_Opacity-5443.png,True,G08340,train,fcc8c3f670a5a3c4e4d18213334df752,d03dd2165e5a7b18
Lung_Opacity-5444.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5444.png,Lung_Opacity/masks/Lung_Opacity-5444.png,True,G08341,train,a9e1992c0fb7af08826ee5179a1edac7,8f602e9c9f591d58
Lung_Opacity-5445.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5445.png,Lung_Opacity/masks/Lung_Opacity-5445.png,True,G08342,train,c752670abe266a5dbe9354a5b657933e,d035970a1435bfb9
Lung_Opacity-5446.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5446.png,Lung_Opacity/masks/Lung_Opacity-5446.png,True,G08343,train,1da78f6c76f381b0fe5fc9172e1d423e,c6783e6b33a3d290
Lung_Opacity-5447.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5447.png,Lung_Opacity/masks/Lung_Opacity-5447.png,True,G08344,test,c94fee4d69b0598576dd98b40b4ed10f,c02d16d04b5d3f9e
Lung_Opacity-5448.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5448.png,Lung_Opacity/masks/Lung_Opacity-5448.png,True,G08345,train,8c23a2c9a75f79f6d36c509d01313da3,992f065e1c5a5d5a
Lung_Opacity-5449.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5449.png,Lung_Opacity/masks/Lung_Opacity-5449.png,True,G08346,train,a908d309b5ef78d88730c5ff6e09ac8e,817817961e175b5b
Lung_Opacity-545.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-545.png,Lung_Opacity/masks/Lung_Opacity-545.png,True,G08347,train,07d7259548adcaf658d695c139ff4ffb,917886161b1d3f3b
Lung_Opacity-5450.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5450.png,Lung_Opacity/masks/Lung_Opacity-5450.png,True,G08348,train,d4600638b35aaa57c81357eb820bae7a,d12bfede5c4840d1
Lung_Opacity-5451.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5451.png,Lung_Opacity/masks/Lung_Opacity-5451.png,True,G08349,train,158981fad2d68574690fef13f390779e,807a47d67f856e90
Lung_Opacity-5452.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5452.png,Lung_Opacity/masks/Lung_Opacity-5452.png,True,G08350,train,8a719e840202217f38b34d398a46221f,c42f43474f0d0d6f
Lung_Opacity-5453.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5453.png,Lung_Opacity/masks/Lung_Opacity-5453.png,True,G08351,train,d1d14cd68705a96107ccd785c1c7540a,dd0a5608092fadfd
Lung_Opacity-5454.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5454.png,Lung_Opacity/masks/Lung_Opacity-5454.png,True,G08352,train,b68530db93cd8bc5bbbb7c6e9b6005cf,c83dfb494c7a3039
Lung_Opacity-5455.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5455.png,Lung_Opacity/masks/Lung_Opacity-5455.png,True,G08353,val,747dfe56326eb576b88ae07a0d6daada,817e861e1d1e595d
Lung_Opacity-5456.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5456.png,Lung_Opacity/masks/Lung_Opacity-5456.png,True,G08354,test,ff2929b8598d0e449fa61b87e94f94fd,c87b15974f1c385a
Lung_Opacity-5457.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5457.png,Lung_Opacity/masks/Lung_Opacity-5457.png,True,G08355,train,53b2e251eb1de26f7e103fbe27041604,892885565a79cdf3
Lung_Opacity-5458.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5458.png,Lung_Opacity/masks/Lung_Opacity-5458.png,True,G08356,val,c7c6adc634751997105d7be30453aad2,c83c4f4b1e493d3c
Lung_Opacity-5459.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5459.png,Lung_Opacity/masks/Lung_Opacity-5459.png,True,G08357,val,0453b69a60377e18d11e6095c81fe39d,816f065a4f4d68ed
Lung_Opacity-546.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-546.png,Lung_Opacity/masks/Lung_Opacity-546.png,True,G08358,test,7d80a842e6e7dad51224556dc4f6bf52,843f5bc13f903bc4
Lung_Opacity-5460.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5460.png,Lung_Opacity/masks/Lung_Opacity-5460.png,True,G08359,train,8c5b9e595103f2a36aa6b6cd9ab7827c,d84d05171a5dbadc
Lung_Opacity-5461.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5461.png,Lung_Opacity/masks/Lung_Opacity-5461.png,True,G08360,train,b79e5e2c2f0a326f0fb1b4520c63e21b,d12943521a5b5f1f
Lung_Opacity-5462.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5462.png,Lung_Opacity/masks/Lung_Opacity-5462.png,True,G08361,train,cc56cdae8ed10c112ffd2fbac605d5b0,9d28071e491f5c5f
Lung_Opacity-5463.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5463.png,Lung_Opacity/masks/Lung_Opacity-5463.png,True,G08362,test,53c9255dd5739938bd0df9e40f0c2d60,ca6cb292ba0fa786
Lung_Opacity-5464.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5464.png,Lung_Opacity/masks/Lung_Opacity-5464.png,True,G08363,test,51330f482388dca14a4a636514a563a2,83608c1f3b1b5d5e
Lung_Opacity-5465.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5465.png,Lung_Opacity/masks/Lung_Opacity-5465.png,True,G08364,train,db6987cf93bb5ba5e1e3dac79e0d38da,dd3807161f591c5b
Lung_Opacity-5466.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5466.png,Lung_Opacity/masks/Lung_Opacity-5466.png,True,G08365,train,d98d7cc2506752ae4150d84ce2811581,cc2ed30f4c1d1e4b
Lung_Opacity-5467.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5467.png,Lung_Opacity/masks/Lung_Opacity-5467.png,True,G08366,train,fc8f4a60ed17d6d2bc70a1cc088809dd,c0356b5a363436b7
Lung_Opacity-5468.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5468.png,Lung_Opacity/masks/Lung_Opacity-5468.png,True,G08367,train,d6ce3a44c1a5f0415524cb911a327dcd,81680e165f797d69
Lung_Opacity-5469.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5469.png,Lung_Opacity/masks/Lung_Opacity-5469.png,True,G08368,train,cee848f4a5ed94a7cc63b1d4708f319d,d53dc60a194f6b38
Lung_Opacity-547.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-547.png,Lung_Opacity/masks/Lung_Opacity-547.png,True,G08369,train,3e1ecda7cdfeaa617ee618ae2bd4eb89,dd2a861f490e5c5d
Lung_Opacity-5470.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5470.png,Lung_Opacity/masks/Lung_Opacity-5470.png,True,G08370,train,9d420d5ef30e710baaebed91d399b481,c42fc50b0f0f6e6c
Lung_Opacity-5471.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5471.png,Lung_Opacity/masks/Lung_Opacity-5471.png,True,G08371,train,aa3122719d826136da2342f3a35ff774,8978861e5a795d4d
Lung_Opacity-5472.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5472.png,Lung_Opacity/masks/Lung_Opacity-5472.png,True,G08372,train,d7ad056ae8b7cbad4696b757f78d290d,de285d5e7a1a381a
Lung_Opacity-5473.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5473.png,Lung_Opacity/masks/Lung_Opacity-5473.png,True,G08373,train,ca3ec1c51313b704017dad577b684d52,d03dc3121d6d3f1c
Lung_Opacity-5474.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5474.png,Lung_Opacity/masks/Lung_Opacity-5474.png,True,G08374,train,5c3b4ce72bcf9a564617fcc1c82e8e33,d42e0f5d4c481f5b
Lung_Opacity-5475.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5475.png,Lung_Opacity/masks/Lung_Opacity-5475.png,True,G08375,val,e7f2658c23b8c2d2773bd736feb7b460,886d075b4d4c6b6b
Lung_Opacity-5476.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5476.png,Lung_Opacity/masks/Lung_Opacity-5476.png,True,G08376,val,9c07f9932602c0109c5004eddbe765ed,d42e4d0b0e2eaeab
Lung_Opacity-5477.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5477.png,Lung_Opacity/masks/Lung_Opacity-5477.png,True,G08377,val,0c6616168a917da8bc340e1bb6203af4,816b202a7f297d3b
Lung_Opacity-5478.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5478.png,Lung_Opacity/masks/Lung_Opacity-5478.png,True,G08378,test,ef76d0898b514a644071d57776a5089c,d47c0b431d1e8d6d
Lung_Opacity-5479.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5479.png,Lung_Opacity/masks/Lung_Opacity-5479.png,True,G08379,train,72a9a895baf8e91301279715b1159e2c,957faaab89e9a081
Lung_Opacity-548.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-548.png,Lung_Opacity/masks/Lung_Opacity-548.png,True,G08380,test,30351b17eae747e620368c0246c175c2,c52a114d4e5f4a7e
Lung_Opacity-5480.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5480.png,Lung_Opacity/masks/Lung_Opacity-5480.png,True,G08381,train,d6d7f3dfbd4e8a3b7ed30ce9086568c2,837e07141f1b5d1d
Lung_Opacity-5481.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5481.png,Lung_Opacity/masks/Lung_Opacity-5481.png,True,G08382,train,9f37d477d5d2af20ac57bb082c7e5311,8a280f5f2a78395f
Lung_Opacity-5482.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5482.png,Lung_Opacity/masks/Lung_Opacity-5482.png,True,G08383,train,8e84d3f814b8dbc4d19f1f7aeec31d26,883d920b0f1f4f1f
Lung_Opacity-5483.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5483.png,Lung_Opacity/masks/Lung_Opacity-5483.png,True,G08384,train,e47372a30795a691efaa602e8eb09f12,cc3e434b8e2d1dac
Lung_Opacity-5484.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5484.png,Lung_Opacity/masks/Lung_Opacity-5484.png,True,G08385,train,342b3d3b9e1cb8330c5bd52e5d8e4505,c07f5f4242594f4d
Lung_Opacity-5485.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5485.png,Lung_Opacity/masks/Lung_Opacity-5485.png,True,G08386,train,ce5da1f56b6e568b5892bf29af7f6ca3,987fc085175b0d1f
Lung_Opacity-5486.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5486.png,Lung_Opacity/masks/Lung_Opacity-5486.png,True,G08387,train,1ed6729776d53525212e24783f97053e,813a4e123a5b3e7b
Lung_Opacity-5487.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5487.png,Lung_Opacity/masks/Lung_Opacity-5487.png,True,G08388,train,a5c685e13cef4a4269846cdfcf9a01d3,cb3478e33ce03ec2
Lung_Opacity-5488.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5488.png,Lung_Opacity/masks/Lung_Opacity-5488.png,True,G08389,train,cc10219118532783d3750aa9f96f8ec7,915a1f7959c879c1
Lung_Opacity-5489.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5489.png,Lung_Opacity/masks/Lung_Opacity-5489.png,True,G08390,train,1c48abdeeb8bd9d5d8149ba9d1f81d75,d53f6a870bc80ccd
Lung_Opacity-549.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-549.png,Lung_Opacity/masks/Lung_Opacity-549.png,True,G08391,train,e0a9a6b3d5fdb83d8056839f5585ea01,9d616a9f7180865f
Lung_Opacity-5490.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5490.png,Lung_Opacity/masks/Lung_Opacity-5490.png,True,G08392,val,6d783cd07d1edbcd919f943c4f0be091,dc09531a1f5e296d
Lung_Opacity-5491.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5491.png,Lung_Opacity/masks/Lung_Opacity-5491.png,True,G08393,train,34dff4bd7bccec4d6099cc8a0f7e3e18,c82f044f6f6163d3
Lung_Opacity-5492.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5492.png,Lung_Opacity/masks/Lung_Opacity-5492.png,True,G08394,train,d2c4927b7d31e89cf14febd35c2c4f03,dd3d52430b464f1b
Lung_Opacity-5493.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5493.png,Lung_Opacity/masks/Lung_Opacity-5493.png,True,G08395,test,46fedd41dd307316800632572d49a332,9968065adc4ae9ee
Lung_Opacity-5494.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5494.png,Lung_Opacity/masks/Lung_Opacity-5494.png,True,G08396,val,fccc12d468b01c0312e88a80d0fea125,8968174e1a557d79
Lung_Opacity-5495.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5495.png,Lung_Opacity/masks/Lung_Opacity-5495.png,True,G08397,val,ff62094ce2b920b378331955b8002014,996fc41255471b5d
Lung_Opacity-5496.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5496.png,Lung_Opacity/masks/Lung_Opacity-5496.png,True,G08398,test,655991b84332335c98989e77f7fe2298,dc3e60413b61733e
Lung_Opacity-5497.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5497.png,Lung_Opacity/masks/Lung_Opacity-5497.png,True,G08399,train,e62de73011ff8c5b72efa99ad2b12ee4,d42e03434e6ee26f
Lung_Opacity-5498.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5498.png,Lung_Opacity/masks/Lung_Opacity-5498.png,True,G08400,train,32a902da1b28ab51dcf2f762ce77422e,c52e5f4b0c0ebb38
Lung_Opacity-5499.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5499.png,Lung_Opacity/masks/Lung_Opacity-5499.png,True,G08401,train,2cb4b7278a0d2a3058b48e2d8d7d1a2d,9139d61e4b3d3949
Lung_Opacity-55.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-55.png,Lung_Opacity/masks/Lung_Opacity-55.png,True,G08402,test,6566b56b854ceba2b6856f2b363096a5,916967c63e956991
Lung_Opacity-550.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-550.png,Lung_Opacity/masks/Lung_Opacity-550.png,True,G08403,train,108fa792703ba40a992d32cca8d69995,947cc7130b4f3d0e
Lung_Opacity-5500.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5500.png,Lung_Opacity/masks/Lung_Opacity-5500.png,True,G08404,train,2f8755da5067194cce71c78eb7f4c405,c03f470b0a6b3f2e
Lung_Opacity-5501.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5501.png,Lung_Opacity/masks/Lung_Opacity-5501.png,True,G08405,train,a9460a7e070402d795d336381aa7b964,956806d61a5a3b7b
Lung_Opacity-5502.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5502.png,Lung_Opacity/masks/Lung_Opacity-5502.png,True,G08406,train,a306875cc90b0484a9c8d45ac2dd5188,99685e392f31a671
Lung_Opacity-5503.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5503.png,Lung_Opacity/masks/Lung_Opacity-5503.png,True,G08407,train,b5417c8255e52d6eb2f99c300dd0bb6d,917a07525e5e5a78
Lung_Opacity-5504.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5504.png,Lung_Opacity/masks/Lung_Opacity-5504.png,True,G08408,train,6c73658686db1642c0cb2c472c04b546,8c7c1f4f0c1f6d22
Lung_Opacity-5505.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5505.png,Lung_Opacity/masks/Lung_Opacity-5505.png,True,G08409,train,b3581641417a3fd5d6a01e614aa613da,d44c070b5d6e6c6e
Lung_Opacity-5506.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5506.png,Lung_Opacity/masks/Lung_Opacity-5506.png,True,G08410,train,1439a76b31c683917f3a7cd9a938d573,883d571e5d4b3d28
Lung_Opacity-5507.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5507.png,Lung_Opacity/masks/Lung_Opacity-5507.png,True,G08411,test,969dbcc2135ba31f47216de7273ad715,817906865f1b5d5e
Lung_Opacity-5508.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5508.png,Lung_Opacity/masks/Lung_Opacity-5508.png,True,G08412,train,f016f68c3fe38d61e2220e96fdc291e6,d47c86c69a8b8d0f
Lung_Opacity-5509.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5509.png,Lung_Opacity/masks/Lung_Opacity-5509.png,True,G08413,train,731d61424b4180bbf2df9df3a0559b8e,dc2d4b0e0c2eadad
Lung_Opacity-551.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-551.png,Lung_Opacity/masks/Lung_Opacity-551.png,True,G08414,train,dc212ff0d5d08c6e9a33d978b5c90b41,dd2c021b017a7d6f
Lung_Opacity-5510.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5510.png,Lung_Opacity/masks/Lung_Opacity-5510.png,True,G08415,val,71189bfba73994e361ccd74bba8ee12a,cd2ee0c2c3456f0f
Lung_Opacity-5511.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5511.png,Lung_Opacity/masks/Lung_Opacity-5511.png,True,G08416,val,591b5b380c50f37399f734e7d32dae23,956ec70f58ec8594
Lung_Opacity-5512.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5512.png,Lung_Opacity/masks/Lung_Opacity-5512.png,True,G08417,val,f4e1f18ce0ea8c99ff5da398a06805e2,91385f581979c9e9
Lung_Opacity-5513.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5513.png,Lung_Opacity/masks/Lung_Opacity-5513.png,True,G08418,test,a44e18af0af18e98c231cc3d7d26518a,c47f434b4b4a4b4b
Lung_Opacity-5514.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5514.png,Lung_Opacity/masks/Lung_Opacity-5514.png,True,G08419,test,e3965b97131cf86b32c5e1a7fb3c7d5e,9520171a3d3d0d7f
Lung_Opacity-5515.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5515.png,Lung_Opacity/masks/Lung_Opacity-5515.png,True,G08420,train,875c1253724e77b8f79f54de1ba89fcd,c02e0e0d0f6d6d6f
Lung_Opacity-5516.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5516.png,Lung_Opacity/masks/Lung_Opacity-5516.png,True,G08421,val,d68b0f3a61b45783df2d5292eb48486e,dc19531b1e28ba97
Lung_Opacity-5517.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5517.png,Lung_Opacity/masks/Lung_Opacity-5517.png,True,G08422,train,7e2fbdd470dc16c2c869d999a5d857ef,d82a4b0c3a3f23b7
Lung_Opacity-5518.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5518.png,Lung_Opacity/masks/Lung_Opacity-5518.png,True,G08423,train,333ccc5ae5e814461ef1438aa8008e45,837016531f7ab4b6
Lung_Opacity-5519.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5519.png,Lung_Opacity/masks/Lung_Opacity-5519.png,True,G08424,train,ef648191a646ee4329c3a55af77560cb,93681f3b497173c2
Lung_Opacity-552.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-552.png,Lung_Opacity/masks/Lung_Opacity-552.png,True,G08425,val,aae0c85f164845cf60b8911feafa1054,9b691086de3af958
Lung_Opacity-5520.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5520.png,Lung_Opacity/masks/Lung_Opacity-5520.png,True,G08426,train,e9dfaedfd2d77f74e83eea2a38d4a6a8,817f941a7b79ea40
Lung_Opacity-5521.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5521.png,Lung_Opacity/masks/Lung_Opacity-5521.png,True,G08427,train,8c8f4e407fb0c2cc4d1be2206da4a15b,897db6d64a4a3a0b
Lung_Opacity-5522.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5522.png,Lung_Opacity/masks/Lung_Opacity-5522.png,True,G08428,train,59e213e020399fd570a0aaaf78f48d06,c57c16181a4f4f2f
Lung_Opacity-5523.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5523.png,Lung_Opacity/masks/Lung_Opacity-5523.png,True,G08429,train,52d9883f1129b1df2b089f97eb947634,912ac4169c5f3b3b
Lung_Opacity-5524.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5524.png,Lung_Opacity/masks/Lung_Opacity-5524.png,True,G08430,val,8bac48b60d1deffd57534fb9ac27c711,c83dc21f1e0b1b6d
Lung_Opacity-5525.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5525.png,Lung_Opacity/masks/Lung_Opacity-5525.png,True,G08431,train,0652caa22ba41503670ea01bd5c4ac08,d42d834b4e4e6e4e
Lung_Opacity-5526.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5526.png,Lung_Opacity/masks/Lung_Opacity-5526.png,True,G08432,train,49d7d4d3828dbb130c757905ad59434b,817b04165a5b7b3b
Lung_Opacity-5527.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5527.png,Lung_Opacity/masks/Lung_Opacity-5527.png,True,G08433,train,72e4985aec5bc8929e62c9d979c6d8fb,813a041e1e1b7f5f
Lung_Opacity-5528.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5528.png,Lung_Opacity/masks/Lung_Opacity-5528.png,True,G08434,train,a7f35ce46b419c5cb7895eb90a2e7b8d,817f161a187b792b
Lung_Opacity-5529.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5529.png,Lung_Opacity/masks/Lung_Opacity-5529.png,True,G08435,test,ceecfac76106574652b463e4b8e1add0,917946061a5f5d7c
Lung_Opacity-553.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-553.png,Lung_Opacity/masks/Lung_Opacity-553.png,True,G08436,train,2ff47ac6b7f85bf34b2d41452f2c4f4a,982a2ec73d092fc7
Lung_Opacity-5530.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5530.png,Lung_Opacity/masks/Lung_Opacity-5530.png,True,G08437,train,a76da0465b67cd66de4fa3fe67b435a0,d52a82d3bbc4add0
Lung_Opacity-5531.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5531.png,Lung_Opacity/masks/Lung_Opacity-5531.png,True,G08438,test,d5a08d72ec937f20d7278fbe56a31a2b,956887578b57a951
Lung_Opacity-5532.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5532.png,Lung_Opacity/masks/Lung_Opacity-5532.png,True,G08439,train,3dae91f2d9d4b98545bcabbdd69e97d3,9728585b6ee8a587
Lung_Opacity-5533.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5533.png,Lung_Opacity/masks/Lung_Opacity-5533.png,True,G08440,train,cee8be8d2b0b86862d476334bf4dfe48,d07e654b3a1f1711
Lung_Opacity-5534.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5534.png,Lung_Opacity/masks/Lung_Opacity-5534.png,True,G08441,train,8b4ebdf54fd28907a1e0773924b12799,dc3d4b0e0e0d2fb1
Lung_Opacity-5535.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5535.png,Lung_Opacity/masks/Lung_Opacity-5535.png,True,G08442,test,1081ab4a8b7c6ce32f14849556fbb857,c83d51195e4f2d2d
Lung_Opacity-5536.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5536.png,Lung_Opacity/masks/Lung_Opacity-5536.png,True,G08443,test,029c6200c020b0d1dd8a6547e95d0e3c,d43d4257521f4e0f
Lung_Opacity-5537.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5537.png,Lung_Opacity/masks/Lung_Opacity-5537.png,True,G08444,train,d4458f33a8dcbaa188b6d9a120094a6b,84685bc30b0f3bbb
Lung_Opacity-5538.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5538.png,Lung_Opacity/masks/Lung_Opacity-5538.png,True,G08445,train,0ac0687c00ca3385e5ddd247a163f2d0,dc2f810d4c7e7a68
Lung_Opacity-5539.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5539.png,Lung_Opacity/masks/Lung_Opacity-5539.png,True,G08446,train,86d1d8e78c63e487de56b4ed8eac6f37,817bc61a5f1a790e
Lung_Opacity-554.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-554.png,Lung_Opacity/masks/Lung_Opacity-554.png,True,G08447,val,b27f6bb0a449830300a69f9fea6277d0,913be0d21ed6565c
Lung_Opacity-5540.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5540.png,Lung_Opacity/masks/Lung_Opacity-5540.png,True,G08448,train,d4f03fa9953ebf94c9cfffca149ddd4b,d13d961d0d1c1d5d
Lung_Opacity-5541.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5541.png,Lung_Opacity/masks/Lung_Opacity-5541.png,True,G08449,train,f79d50b4cda9d47ad40350dbe23e2d44,d538481e7e1b7c25
Lung_Opacity-5542.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5542.png,Lung_Opacity/masks/Lung_Opacity-5542.png,True,G08450,train,932501126f285345799f44c5e457ac46,943ec54f434b4f0d
Lung_Opacity-5543.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5543.png,Lung_Opacity/masks/Lung_Opacity-5543.png,True,G08451,val,dec42b112731a4809dce37025f400e58,9959c61e190baf2d
Lung_Opacity-5544.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5544.png,Lung_Opacity/masks/Lung_Opacity-5544.png,True,G08452,train,73124fa01df1b5f650b75bbd1159a8b6,ff7c9539aba0a082
Lung_Opacity-5545.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5545.png,Lung_Opacity/masks/Lung_Opacity-5545.png,True,G08453,train,83d092ff7fd8aff8a9a4f00f7b028eba,cc4f050f0f2b0d3f
Lung_Opacity-5546.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5546.png,Lung_Opacity/masks/Lung_Opacity-5546.png,True,G08454,val,10fbbb13dcdc0076d2c376ddce368edf,c50e534b4d4a4f4f
Lung_Opacity-5547.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5547.png,Lung_Opacity/masks/Lung_Opacity-5547.png,True,G08455,train,b213c6476bba89c1139bc79b770fac39,c83d430b1d0f6d3d
Lung_Opacity-5548.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5548.png,Lung_Opacity/masks/Lung_Opacity-5548.png,True,G08456,train,77fb624b316a1d5c326bd21730cc853f,96681f4b2a6e6c6c
Lung_Opacity-5549.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5549.png,Lung_Opacity/masks/Lung_Opacity-5549.png,True,G08457,train,92aaa9c3e99036365dcc30b3dacee48e,8978871a1d4b78ed
Lung_Opacity-555.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-555.png,Lung_Opacity/masks/Lung_Opacity-555.png,True,G08458,test,26a408a43ded1ddfbc624f0b50ea5c27,913dc7561a59475c
Lung_Opacity-5550.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5550.png,Lung_Opacity/masks/Lung_Opacity-5550.png,True,G08459,test,e156e4ffb0a978ea8d2c8c74833404c3,d47ea5528b4e8a4e
Lung_Opacity-5551.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5551.png,Lung_Opacity/masks/Lung_Opacity-5551.png,True,G08460,train,addc1fe16f5af19ebcf278e12b907287,c274364f2f2c2e2e
Lung_Opacity-5552.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5552.png,Lung_Opacity/masks/Lung_Opacity-5552.png,True,G08461,train,80e717e69e01d665c2ae3e41a4fb60a1,9168061e5a796d6f
Lung_Opacity-5553.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5553.png,Lung_Opacity/masks/Lung_Opacity-5553.png,True,G08462,train,ebff253a52ae4f96bd163c266e7876ef,fa611e1a785a7961
Lung_Opacity-5554.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5554.png,Lung_Opacity/masks/Lung_Opacity-5554.png,True,G08463,train,082173f42e917cef45e8c4da4bf2eee0,d42d75c34bc969c8
Lung_Opacity-5555.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5555.png,Lung_Opacity/masks/Lung_Opacity-5555.png,True,G08464,train,bc6b558d090e9c695c7c3275ee7cebc2,996807160e7d7979
Lung_Opacity-5556.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5556.png,Lung_Opacity/masks/Lung_Opacity-5556.png,True,G08465,train,7ed41492969c8f9e3ed36ba490345dc1,917b101e5a58797b
Lung_Opacity-5557.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5557.png,Lung_Opacity/masks/Lung_Opacity-5557.png,True,G08466,train,77c1cf030e32cd2e4e5bc7631b54a07e,cd0e830b0d5d4f5d
Lung_Opacity-5558.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5558.png,Lung_Opacity/masks/Lung_Opacity-5558.png,True,G08467,train,3d7bd9cc320b1161f18c318bf86a91a1,d03b56025ec25b7b
Lung_Opacity-5559.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5559.png,Lung_Opacity/masks/Lung_Opacity-5559.png,True,G08468,train,66868d28dbdf3b46a105386de0ef2ec4,9978471a4f4a1e9b
Lung_Opacity-556.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-556.png,Lung_Opacity/masks/Lung_Opacity-556.png,True,G08469,train,17e15e8da3c1eae5b5cd1c9eb01c3bdd,907a521a591b1f3f
Lung_Opacity-5560.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5560.png,Lung_Opacity/masks/Lung_Opacity-5560.png,True,G08470,train,ea8a0a189d2ff0e83b2df03579196e14,913a1a1b1b7b3b1a
Lung_Opacity-5561.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5561.png,Lung_Opacity/masks/Lung_Opacity-5561.png,True,G08471,train,adc339906f10cd13e16d87330e901f04,c97cc60f0f1d490f
Lung_Opacity-5562.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5562.png,Lung_Opacity/masks/Lung_Opacity-5562.png,True,G08472,test,007dfac4073a28aa7e1a2f28323ae8c9,8c44d74b0d5d1d1f
Lung_Opacity-5563.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5563.png,Lung_Opacity/masks/Lung_Opacity-5563.png,True,G08473,test,a3b0470c859ae13e928b42e5bc64f108,c93e161a4f4d68e9
Lung_Opacity-5564.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5564.png,Lung_Opacity/masks/Lung_Opacity-5564.png,True,G08474,train,85beec79216f292010294f2138fc0026,81785e1e1a5f7925
Lung_Opacity-5565.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5565.png,Lung_Opacity/masks/Lung_Opacity-5565.png,True,G08475,train,be7420c978d3f67c2b37379418332d6a,95691c7a6b616363
Lung_Opacity-5566.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5566.png,Lung_Opacity/masks/Lung_Opacity-5566.png,True,G08476,val,33d488b2818094cf2361701cac5b875c,93780c197b5959cb
Lung_Opacity-5567.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5567.png,Lung_Opacity/masks/Lung_Opacity-5567.png,True,G08477,test,d22e6962663d790806445e715e78c79b,c4bc4943cb4b4b2f
Lung_Opacity-5568.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5568.png,Lung_Opacity/masks/Lung_Opacity-5568.png,True,G08478,val,9dbcf68d4094d44a0e6df899240ab307,8361789a799b7989
Lung_Opacity-5569.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5569.png,Lung_Opacity/masks/Lung_Opacity-5569.png,True,G08479,val,bdd9c033e4c12528c96d83b97cb0ef0c,d029db5f48181e3f
Lung_Opacity-557.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-557.png,Lung_Opacity/masks/Lung_Opacity-557.png,True,G08480,train,167f8f4740d5c6e285976193bbcb533c,cc3ed30b4c4e1f4c
Lung_Opacity-5570.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5570.png,Lung_Opacity/masks/Lung_Opacity-5570.png,True,G08481,train,14a1f78fe0096e99b86b191375854601,8c64070f4e7e6d6c
Lung_Opacity-5571.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5571.png,Lung_Opacity/masks/Lung_Opacity-5571.png,True,G08482,train,732e7992f3030f0eded9e7edad52a33a,997b85161f0d5e19
Lung_Opacity-5572.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5572.png,Lung_Opacity/masks/Lung_Opacity-5572.png,True,G08483,train,2a3f5bfc413c8254cf4f352083951254,d42e854aecccd3d1
Lung_Opacity-5573.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5573.png,Lung_Opacity/masks/Lung_Opacity-5573.png,True,G08484,train,776d7fc196472ef1d221485f7213c72d,c89d3fa23462ce2d
Lung_Opacity-5574.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5574.png,Lung_Opacity/masks/Lung_Opacity-5574.png,True,G08485,train,e5a44712faa561e72d8ed304dbb4d902,954a4e5f7aeb4828
Lung_Opacity-5575.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5575.png,Lung_Opacity/masks/Lung_Opacity-5575.png,True,G08486,train,efa4960b8c3281d3d2bbccda605d65e8,c225cb4f6e383cb1
Lung_Opacity-5576.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5576.png,Lung_Opacity/masks/Lung_Opacity-5576.png,True,G08487,train,97c36183bc20d71612f618d22ffdaad1,c03d4c0f1d3d9d95
Lung_Opacity-5577.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5577.png,Lung_Opacity/masks/Lung_Opacity-5577.png,True,G08488,val,ce78a75c4492e0899fd3572085cd3a0c,9578045e79686dec
Lung_Opacity-5578.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5578.png,Lung_Opacity/masks/Lung_Opacity-5578.png,True,G08489,train,465d2e79080b0d191330901b98c49310,993aae1b0d5859cb
Lung_Opacity-5579.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5579.png,Lung_Opacity/masks/Lung_Opacity-5579.png,True,G08490,test,e02ea3947dbb9e7cd45443b1883e0bc4,9574960d9d3e6c94
Lung_Opacity-558.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-558.png,Lung_Opacity/masks/Lung_Opacity-558.png,True,G08491,test,d9bff1db78d3591beabe4a8b24298826,817faa0e0f0f3d0d
Lung_Opacity-5580.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5580.png,Lung_Opacity/masks/Lung_Opacity-5580.png,True,G08492,train,d23861fa05ebef627c181ef5054ce304,946e071d0c7c3a7d
Lung_Opacity-5581.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5581.png,Lung_Opacity/masks/Lung_Opacity-5581.png,True,G08493,train,8996a1f202fe8b967e59ad80b0603526,987b471a49293da7
Lung_Opacity-5582.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5582.png,Lung_Opacity/masks/Lung_Opacity-5582.png,True,G08494,train,149fc0c9979150b433eec1a409cd3e30,912d86171e1e1f5e
Lung_Opacity-5583.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5583.png,Lung_Opacity/masks/Lung_Opacity-5583.png,True,G08495,train,c69e1955a7ae7641f4b19469c22632b6,d42a1b1f1a3aa8f9
Lung_Opacity-5584.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5584.png,Lung_Opacity/masks/Lung_Opacity-5584.png,True,G08496,train,e4d79e15441a95bce3b2760829fcfb96,c02c0e5e1b6f4d7c
Lung_Opacity-5585.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5585.png,Lung_Opacity/masks/Lung_Opacity-5585.png,True,G08497,train,656268680c9d76635c751a1357b39ff1,992bd61ed65c5941
Lung_Opacity-5586.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5586.png,Lung_Opacity/masks/Lung_Opacity-5586.png,True,G08498,val,4ba20ea3aa12052004cf4f3db57cd238,d17ba0164e5c3e1b
Lung_Opacity-5587.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5587.png,Lung_Opacity/masks/Lung_Opacity-5587.png,True,G08499,val,cea957ee06f078b5b6812645b5056e9b,d42f35511b5e4c2d
Lung_Opacity-5588.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5588.png,Lung_Opacity/masks/Lung_Opacity-5588.png,True,G08500,test,7ab20b59daf390cd84e13c7ba75b9fb3,d57f350a0b2d2c3c
Lung_Opacity-5589.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5589.png,Lung_Opacity/masks/Lung_Opacity-5589.png,True,G08501,train,fb54489312ff1c3e5c8d0c35faec6423,dd2853c3535e494d
Lung_Opacity-559.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-559.png,Lung_Opacity/masks/Lung_Opacity-559.png,True,G08502,train,043ef2353bd47e72d9733351cbaaa284,c12916165a5f7d4b
Lung_Opacity-5590.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5590.png,Lung_Opacity/masks/Lung_Opacity-5590.png,True,G08503,train,b125aa0472ab6493647809a72b4f44c7,c92d164b193e5a6b
Lung_Opacity-5591.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5591.png,Lung_Opacity/masks/Lung_Opacity-5591.png,True,G08504,train,ba0d43833bf10637095d73515d335065,d93fc61ec0796069
Lung_Opacity-5592.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5592.png,Lung_Opacity/masks/Lung_Opacity-5592.png,True,G08505,train,5f893966541a5c0bb522b7f6a89a9276,c92b560a1d2b2fae
Lung_Opacity-5593.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5593.png,Lung_Opacity/masks/Lung_Opacity-5593.png,True,G08506,train,8f7e08e96ef020afea21e5bd2c4595b0,912a07171e5f1f1d
Lung_Opacity-5594.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5594.png,Lung_Opacity/masks/Lung_Opacity-5594.png,True,G08507,train,f50f854fba87fbc8b483db645f883350,d42ec24d871eac57
Lung_Opacity-5595.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5595.png,Lung_Opacity/masks/Lung_Opacity-5595.png,True,G08508,train,389b466c873a8ef4b36767d28119d0bf,c42d474f0d6c6c7c
Lung_Opacity-5596.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5596.png,Lung_Opacity/masks/Lung_Opacity-5596.png,True,G08509,train,3ec96a22f98e92195bf18af07351ee62,897f1506171e5d1d
Lung_Opacity-5597.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5597.png,Lung_Opacity/masks/Lung_Opacity-5597.png,True,G08510,train,1089953435c1496db19124b3a747f2ff,c02b17965e1d397a
Lung_Opacity-5598.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5598.png,Lung_Opacity/masks/Lung_Opacity-5598.png,True,G08511,train,8ad74ce93919fb5f8133c1c6289ebe64,81611f0f1e171f1f
Lung_Opacity-5599.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5599.png,Lung_Opacity/masks/Lung_Opacity-5599.png,True,G08512,train,1237f081e420115ca727a880ae33b0c8,c06b15975e0c1e7b
Lung_Opacity-56.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-56.png,Lung_Opacity/masks/Lung_Opacity-56.png,True,G08513,val,82c5d07d15b2b7352c1047a770c66cea,803d160f1f1f1d5e
Lung_Opacity-560.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-560.png,Lung_Opacity/masks/Lung_Opacity-560.png,True,G08514,train,d685c697795a8cd58962202317e1c2c3,cd3d520b410f1e6f
Lung_Opacity-5600.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5600.png,Lung_Opacity/masks/Lung_Opacity-5600.png,True,G08515,test,b4b76ca431e0bb7c1a66722d3a4cd4fa,c42ed5086e2f6d3c
Lung_Opacity-5601.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5601.png,Lung_Opacity/masks/Lung_Opacity-5601.png,True,G08516,test,63f7db805661438c832b36bdc7a509b3,981a531e1f2ab9ba
Lung_Opacity-5602.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5602.png,Lung_Opacity/masks/Lung_Opacity-5602.png,True,G08517,val,cb581f579fe385155bde0a18565852bc,9568141e4f5c597d
Lung_Opacity-5603.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5603.png,Lung_Opacity/masks/Lung_Opacity-5603.png,True,G08518,test,da21eaaa6c51f3e15ee4930aeebb4bbd,8528071a1f5f4f5d
Lung_Opacity-5604.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5604.png,Lung_Opacity/masks/Lung_Opacity-5604.png,True,G08519,train,6455eb936145f6d08b933ef2617d77bd,907fef465a1a1a1a
Lung_Opacity-5605.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5605.png,Lung_Opacity/masks/Lung_Opacity-5605.png,True,G08520,train,1e0cfa06d19c9ca07cdc49480fc16e14,936acf9c504b4373
Lung_Opacity-5606.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5606.png,Lung_Opacity/masks/Lung_Opacity-5606.png,True,G08521,val,956d0b69e47f0cfd531cc7b336a6d487,d02b431e3c647577
Lung_Opacity-5607.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5607.png,Lung_Opacity/masks/Lung_Opacity-5607.png,True,G08522,train,290f0f0fe84eeef838a7e08600ebcff9,d52ad1cb4d4c4d8b
Lung_Opacity-5608.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5608.png,Lung_Opacity/masks/Lung_Opacity-5608.png,True,G08523,train,240db1d86ca8da1e97ab243b694c8b72,c42fc74b0d4c6c6d
Lung_Opacity-5609.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5609.png,Lung_Opacity/masks/Lung_Opacity-5609.png,True,G08524,train,f625e525f547df8b5c5694880d26ef3c,dc2e425bc31f504f
Lung_Opacity-561.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-561.png,Lung_Opacity/masks/Lung_Opacity-561.png,True,G08525,train,797d7582359085ef3e2e3820c7efeb4d,956a36a19c6ba639
Lung_Opacity-5610.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5610.png,Lung_Opacity/masks/Lung_Opacity-5610.png,True,G08526,train,dcc6251863f2f53d03147e51ce80e156,817f8a5a494d475e
Lung_Opacity-5611.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5611.png,Lung_Opacity/masks/Lung_Opacity-5611.png,True,G08527,train,0579644adc6d9b242486c282dd7d70e6,9969061e5b5c7879
Lung_Opacity-5612.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5612.png,Lung_Opacity/masks/Lung_Opacity-5612.png,True,G08528,train,6c332b52851656c989b13c2d775f8700,dd2893cb0c1cacdd
Lung_Opacity-5613.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5613.png,Lung_Opacity/masks/Lung_Opacity-5613.png,True,G08529,train,3d9e6c48cf31ee1fb2f4d74cfbf7f36c,d42cd15b4b4b1a4f
Lung_Opacity-5614.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5614.png,Lung_Opacity/masks/Lung_Opacity-5614.png,True,G08530,train,f1cf2e9cfbf2686979591f5bbf63fae0,987947c65e1c4d4d
Lung_Opacity-5615.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5615.png,Lung_Opacity/masks/Lung_Opacity-5615.png,True,G08531,train,c70c76c62fe8e41cd8a5e130fde0784c,dd2c534b0e0d1f2e
Lung_Opacity-5616.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5616.png,Lung_Opacity/masks/Lung_Opacity-5616.png,True,G08532,train,0e4603fa65e39c54c0a3e125421b93b3,817b149e1a1bfb51
Lung_Opacity-5617.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5617.png,Lung_Opacity/masks/Lung_Opacity-5617.png,True,G08533,train,d19aa37dac358d910c2d037384688b61,dd28d2d3564f4a68
Lung_Opacity-5618.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5618.png,Lung_Opacity/masks/Lung_Opacity-5618.png,True,G08534,train,3cada9501d04255e5fc7f5893dd439d1,817f541a574b194f
Lung_Opacity-5619.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5619.png,Lung_Opacity/masks/Lung_Opacity-5619.png,True,G08535,val,78c948489ed4b456ad712310f7e67ac8,d42d5b4b4b4d0d0f
Lung_Opacity-562.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-562.png,Lung_Opacity/masks/Lung_Opacity-562.png,True,G08536,val,3e6a5c43ded75f007e87cb662aa1e3b0,d42d534b0f0d3d3c
Lung_Opacity-5620.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5620.png,Lung_Opacity/masks/Lung_Opacity-5620.png,True,G08537,train,a3769811820d96f1b91138e600ec1e5b,94ec871b541d3d0f
Lung_Opacity-5621.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5621.png,Lung_Opacity/masks/Lung_Opacity-5621.png,True,G08538,test,44ed368f8400941d483455c1bde57c16,817bc01e59796b6a
Lung_Opacity-5622.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5622.png,Lung_Opacity/masks/Lung_Opacity-5622.png,True,G08539,val,9fd82616c37076ce23776c1ea17d4207,981b121e78f86de5
Lung_Opacity-5623.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5623.png,Lung_Opacity/masks/Lung_Opacity-5623.png,True,G08540,test,f49a7c3f4c67cc3e13c5c89e7e0a2541,c03d4b0a2c6f3f96
Lung_Opacity-5624.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5624.png,Lung_Opacity/masks/Lung_Opacity-5624.png,True,G08541,test,ea0b600ead2ef0068f8f3721ca4524ca,956a071e5c1d68ed
Lung_Opacity-5625.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5625.png,Lung_Opacity/masks/Lung_Opacity-5625.png,True,G08542,train,02ac14c19ec9b80030b833d139f754f5,917b0595969e1e1d
Lung_Opacity-5626.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5626.png,Lung_Opacity/masks/Lung_Opacity-5626.png,True,G08543,train,e10b6d63d1e78f95b254dc0ddf241139,d52f404f0e2d37a5
Lung_Opacity-5627.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5627.png,Lung_Opacity/masks/Lung_Opacity-5627.png,True,G08544,train,3a63804f4abac40e9fa84c349a05fc60,d52b055d0f0e4f0f
Lung_Opacity-5628.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5628.png,Lung_Opacity/masks/Lung_Opacity-5628.png,True,G08545,train,e7e39fc3f37cbe64b215b9849d9f1215,d03f080d59395be7
Lung_Opacity-5629.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5629.png,Lung_Opacity/masks/Lung_Opacity-5629.png,True,G08546,val,134fa1759d92e95554807ced8b0a14fa,c26a63682575f393
Lung_Opacity-563.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-563.png,Lung_Opacity/masks/Lung_Opacity-563.png,True,G08547,train,7bb544ec0208a7e682baad2da671ba6e,9169c21ec71e5e5a
Lung_Opacity-5630.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5630.png,Lung_Opacity/masks/Lung_Opacity-5630.png,True,G08548,train,0e75e711d247d4cd0450632f6d9bb7e5,d92b87161a595c5b
Lung_Opacity-5631.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5631.png,Lung_Opacity/masks/Lung_Opacity-5631.png,True,G08549,val,1b6d1b395c932802916c5568a78fe9c8,d02d4d0b0c2f2ebf
Lung_Opacity-5632.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5632.png,Lung_Opacity/masks/Lung_Opacity-5632.png,True,G08550,train,e1dc4af2160af1adae20fb178d3350fb,942f435374f13237
Lung_Opacity-5633.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5633.png,Lung_Opacity/masks/Lung_Opacity-5633.png,True,G08551,train,9215e9d757415e8a50bf45d2e67a34f8,9c1a930e2a3b7bf0
Lung_Opacity-5634.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5634.png,Lung_Opacity/masks/Lung_Opacity-5634.png,True,G08552,train,a8133b74dc6e8c69195bbc4251a1abe9,9d0e071b4d5c69ea
Lung_Opacity-5635.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5635.png,Lung_Opacity/masks/Lung_Opacity-5635.png,True,G08553,train,92b11dc6a7361609e9c387dde81fc86c,c83d02574b5e0b5f
Lung_Opacity-5636.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5636.png,Lung_Opacity/masks/Lung_Opacity-5636.png,True,G08554,train,3910e7b65d0ce35e173ec68c0a8f092f,9179461e1a3b293f
Lung_Opacity-5637.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5637.png,Lung_Opacity/masks/Lung_Opacity-5637.png,True,G08555,val,49d96795cea830dcadcac0e5df9d83ac,d93ad21b392839ab
Lung_Opacity-5638.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5638.png,Lung_Opacity/masks/Lung_Opacity-5638.png,True,G08556,test,1707752ec0ebc47b21f4741b65d96013,856820571f5b5f1b
Lung_Opacity-5639.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5639.png,Lung_Opacity/masks/Lung_Opacity-5639.png,True,G08557,train,12105b93eeadcf6b3a388ed861e19559,dd29d20b5f4d481b
Lung_Opacity-564.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-564.png,Lung_Opacity/masks/Lung_Opacity-564.png,True,G08558,test,3fb2a20360f773b2ff1e76b722313ed3,d918d20a3ce9bdb8
Lung_Opacity-5640.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5640.png,Lung_Opacity/masks/Lung_Opacity-5640.png,True,G08559,train,d5da9ac92e0150794451ae1f53e3b168,d52e07171e1e5c5c
Lung_Opacity-5641.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5641.png,Lung_Opacity/masks/Lung_Opacity-5641.png,True,G08560,test,ec13e219679907cab8845f356aa57c34,91301f3d39b5a9c5
Lung_Opacity-5642.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5642.png,Lung_Opacity/masks/Lung_Opacity-5642.png,True,G08561,train,d6932b19e8ea6ed7004193b8ec026a69,91685e1b5b794b29
Lung_Opacity-5643.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5643.png,Lung_Opacity/masks/Lung_Opacity-5643.png,True,G08562,test,1631bf6af4a84c167bc29338330c21e2,895a0c5e5a5be969
Lung_Opacity-5644.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5644.png,Lung_Opacity/masks/Lung_Opacity-5644.png,True,G08563,train,9135b116b1394f3a0266f7c73c3ee108,d07f5243164f4b1b
Lung_Opacity-5645.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5645.png,Lung_Opacity/masks/Lung_Opacity-5645.png,True,G08564,train,57b4c5b72b67de84cf01a8467bb6f93d,c03f0b39b0d2cbcd
Lung_Opacity-5646.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5646.png,Lung_Opacity/masks/Lung_Opacity-5646.png,True,G08565,train,2c58e075e31e4587560b0870a73bfc4f,9e613a2974a2e7c6
Lung_Opacity-5647.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5647.png,Lung_Opacity/masks/Lung_Opacity-5647.png,True,G08566,train,00bc1d83f9f898bdbe5bb34b9b7ed64a,c87d173d6f626460
Lung_Opacity-5648.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5648.png,Lung_Opacity/masks/Lung_Opacity-5648.png,True,G08567,train,048356e3f76babebd3c2457d9353cc98,9d38140f1b4d1f4f
Lung_Opacity-5649.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5649.png,Lung_Opacity/masks/Lung_Opacity-5649.png,True,G08568,train,8fb21f699e27e60f3c1c087e403cd565,c02fcb4c2a277733
Lung_Opacity-565.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-565.png,Lung_Opacity/masks/Lung_Opacity-565.png,True,G08569,train,c198a6c29a4eea4fcb27b40bb709641e,942d46d13ec46bdc
Lung_Opacity-5650.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5650.png,Lung_Opacity/masks/Lung_Opacity-5650.png,True,G08570,train,302853a60c0ae2828b31be96353c1fb0,d03b4b187a0b393f
Lung_Opacity-5651.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5651.png,Lung_Opacity/masks/Lung_Opacity-5651.png,True,G08571,train,2fe0d76f2ca09de04f6fabee0ddea63c,902fc51f1c0f1e5e
Lung_Opacity-5652.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5652.png,Lung_Opacity/masks/Lung_Opacity-5652.png,True,G08572,train,1370d859e50f865f9ddeedb79a1118ec,d43f43474b0f1d0e
Lung_Opacity-5653.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5653.png,Lung_Opacity/masks/Lung_Opacity-5653.png,True,G08573,train,a8695715fb5490ea36cfda99f720947e,dc2dc31f0d0d1d4b
Lung_Opacity-5654.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5654.png,Lung_Opacity/masks/Lung_Opacity-5654.png,True,G08574,train,9dcce5c6303eedfc366acba1c46f3324,91784616195f1f1f
Lung_Opacity-5655.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5655.png,Lung_Opacity/masks/Lung_Opacity-5655.png,True,G08575,train,b39cdf79e779526a9020a818e59c938d,943d574b4e0e5f0a
Lung_Opacity-5656.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5656.png,Lung_Opacity/masks/Lung_Opacity-5656.png,True,G08576,val,8eff54261653c2f6bd77d46cfcc9ddb3,913a5a1a5b5b13d3
Lung_Opacity-5657.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5657.png,Lung_Opacity/masks/Lung_Opacity-5657.png,True,G08577,train,56ad5b7b44caf0812df541a4db1df4c1,cd1c4a0b2d2da5f5
Lung_Opacity-5658.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5658.png,Lung_Opacity/masks/Lung_Opacity-5658.png,True,G08578,val,274b8e0d4ff90f85be84528d4a298e99,8c6f821b4d4d5e5c
Lung_Opacity-5659.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5659.png,Lung_Opacity/masks/Lung_Opacity-5659.png,True,G08579,val,b8571f98903feeab9e0d317d33d03c31,c43d12135b5e4f1b
Lung_Opacity-566.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-566.png,Lung_Opacity/masks/Lung_Opacity-566.png,True,G08580,test,6a5ec44b7c999d02daf537f1a07e1a8d,d03502173e3a3f3d
Lung_Opacity-5660.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5660.png,Lung_Opacity/masks/Lung_Opacity-5660.png,True,G08581,train,ef989106d146f849faab473ca695954c,81191a7b38a3b797
Lung_Opacity-5661.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5661.png,Lung_Opacity/masks/Lung_Opacity-5661.png,True,G08582,test,f922efbe167da55da7bb52bd4262647d,913a1e5bdec294a6
Lung_Opacity-5662.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5662.png,Lung_Opacity/masks/Lung_Opacity-5662.png,True,G08583,train,c6786b0a0e723088e0754d4995f64ad9,d43d430b1f0d1f6c
Lung_Opacity-5663.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5663.png,Lung_Opacity/masks/Lung_Opacity-5663.png,True,G08584,train,c06b13a65259814b04bf19c51d35ad1c,9d2c431a4d4f4e4f
Lung_Opacity-5664.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5664.png,Lung_Opacity/masks/Lung_Opacity-5664.png,True,G08585,train,67d3a4787396c03b41d1a9fff57e4327,956806061f5f1d3f
Lung_Opacity-5665.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5665.png,Lung_Opacity/masks/Lung_Opacity-5665.png,True,G08586,train,305932bcd5cc0f9e0236e2c8b5b17d3e,dc2cd14b0e0f0f1f
Lung_Opacity-5666.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5666.png,Lung_Opacity/masks/Lung_Opacity-5666.png,True,G08587,train,ac79ecfecf1a0be6e570315aab8109f2,d52a43471f4e590f
Lung_Opacity-5667.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5667.png,Lung_Opacity/masks/Lung_Opacity-5667.png,True,G08588,train,798f3ea2f84504f1a81e56114ef8fc09,9128061e1f4f1f5f
Lung_Opacity-5668.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5668.png,Lung_Opacity/masks/Lung_Opacity-5668.png,True,G08589,train,e598292a11f405fc2b9db7df8ed25623,d03be3695c456939
Lung_Opacity-5669.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5669.png,Lung_Opacity/masks/Lung_Opacity-5669.png,True,G08590,train,689998afe9c7835a04e7a6064114d862,d52ed24153534b4f
Lung_Opacity-567.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-567.png,Lung_Opacity/masks/Lung_Opacity-567.png,True,G08591,train,91db1f47f7f3b364eee8cb4ba644a06a,813f1612185b5f5f
Lung_Opacity-5670.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5670.png,Lung_Opacity/masks/Lung_Opacity-5670.png,True,G08592,train,11f65e0392efdcfd3b7e195c6fffe8c8,c97d961658417979
Lung_Opacity-5671.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5671.png,Lung_Opacity/masks/Lung_Opacity-5671.png,True,G08593,train,48f0082fad7f236e3c99bacc66163022,cd3ed24b174b0d2c
Lung_Opacity-5672.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5672.png,Lung_Opacity/masks/Lung_Opacity-5672.png,True,G08594,val,7c72151af3c9fedfc549a69f0f8770d7,c82d1703db5f015f
Lung_Opacity-5673.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5673.png,Lung_Opacity/masks/Lung_Opacity-5673.png,True,G08595,train,1d4bc821831b493aaefe8149d47f3bfe,8829c682df9dbd92
Lung_Opacity-5674.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5674.png,Lung_Opacity/masks/Lung_Opacity-5674.png,True,G08596,test,ec5a06cbea14bb6b3350b37789ec64f2,903e8d1d577a6a62
Lung_Opacity-5675.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5675.png,Lung_Opacity/masks/Lung_Opacity-5675.png,True,G08597,train,15629e0c6eb6660306d84a65d7ab0d0f,d92a5e5d4d0e0b0f
Lung_Opacity-5676.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5676.png,Lung_Opacity/masks/Lung_Opacity-5676.png,True,G08598,val,2c0535ee8155097d5989e13a10b2e94c,897d87961a1b1f1a
Lung_Opacity-5677.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5677.png,Lung_Opacity/masks/Lung_Opacity-5677.png,True,G08599,test,61d29da8ebacb337618cb426d548391e,d03f99180e7a6a7a
Lung_Opacity-5678.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5678.png,Lung_Opacity/masks/Lung_Opacity-5678.png,True,G08600,test,ccd79bdf7de2e52526ea2e64d7505e89,c83d524b4f0f693c
Lung_Opacity-5679.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5679.png,Lung_Opacity/masks/Lung_Opacity-5679.png,True,G08601,train,477a60adaf0f005c8b6b31a2dbb87305,957c8e181d3d1d1d
Lung_Opacity-568.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-568.png,Lung_Opacity/masks/Lung_Opacity-568.png,True,G08602,train,887678713805084f455de94d38117d0a,d62dd14e0e4a495f
Lung_Opacity-5680.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5680.png,Lung_Opacity/masks/Lung_Opacity-5680.png,True,G08603,test,60f0ce2825700397162cb7bda68058f0,d624570a0f1a5f3b
Lung_Opacity-5681.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5681.png,Lung_Opacity/masks/Lung_Opacity-5681.png,True,G08604,val,44b0f86f447cff7724ef5bc5e11fce58,942b4f1e587a68e9
Lung_Opacity-5682.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5682.png,Lung_Opacity/masks/Lung_Opacity-5682.png,True,G08605,train,4af232a7457dee98e28e22952b889736,983f461a596969e9
Lung_Opacity-5683.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5683.png,Lung_Opacity/masks/Lung_Opacity-5683.png,True,G08606,train,8d13893e1d11dfa72ef8d3c1f0919001,c52f474b0d4a4a6f
Lung_Opacity-5684.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5684.png,Lung_Opacity/masks/Lung_Opacity-5684.png,True,G08607,train,35da5a13e083e3dce30fa93385291e94,dc5d534b5d4a1c0e
Lung_Opacity-5685.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5685.png,Lung_Opacity/masks/Lung_Opacity-5685.png,True,G08608,train,41db105169810f2630d1946e3b67f924,917a40195b5f5e2e
Lung_Opacity-5686.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5686.png,Lung_Opacity/masks/Lung_Opacity-5686.png,True,G08609,train,4cd00567775fce3e05617b2bc0c06dde,917bc61859e9f8c2
Lung_Opacity-5687.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5687.png,Lung_Opacity/masks/Lung_Opacity-5687.png,True,G08610,train,1b0a9b45b8a6e417b05b33d93f287bf3,ce3d49c9490d6d0f
Lung_Opacity-5688.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5688.png,Lung_Opacity/masks/Lung_Opacity-5688.png,True,G08611,train,6d9addbc8b726fe82df8c63b81b95bf8,d86f461f59081d3d
Lung_Opacity-5689.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5689.png,Lung_Opacity/masks/Lung_Opacity-5689.png,True,G08612,train,f427c4a661edfe1e9c4bb11bf2a7e3f4,95682b837a976695
Lung_Opacity-569.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-569.png,Lung_Opacity/masks/Lung_Opacity-569.png,True,G08613,train,887f4c6c529d9e66441661d071c58dde,8c6f030d0f0c2fef
Lung_Opacity-5690.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5690.png,Lung_Opacity/masks/Lung_Opacity-5690.png,True,G08614,train,2a70bc9094e3883a78372218340e0ef7,912f84183d7b7867
Lung_Opacity-5691.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5691.png,Lung_Opacity/masks/Lung_Opacity-5691.png,True,G08615,train,fef8974e7ffed9c59258f7da75b38c7e,dd0a1b0b1d1c3eba
Lung_Opacity-5692.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5692.png,Lung_Opacity/masks/Lung_Opacity-5692.png,True,G08616,train,17480279c5f71583fface45a9a604b74,812b5487961e5f1f
Lung_Opacity-5693.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5693.png,Lung_Opacity/masks/Lung_Opacity-5693.png,True,G08617,train,de7728e27cf7e0ef1069a874e5f89dea,dc2e431b0f5a1c4f
Lung_Opacity-5694.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5694.png,Lung_Opacity/masks/Lung_Opacity-5694.png,True,G08618,val,486a5984651fc7551d0c03f01b81010e,cc0c834f4e4d6fe1
Lung_Opacity-5695.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5695.png,Lung_Opacity/masks/Lung_Opacity-5695.png,True,G08619,val,543277e5faf49f51815531c0c9ccf23d,d42c41c30a3f0fef
Lung_Opacity-5696.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5696.png,Lung_Opacity/masks/Lung_Opacity-5696.png,True,G08620,val,e7cc874f23efbaffb0ee3b02710b7130,903d05153d6f7a63
Lung_Opacity-5697.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5697.png,Lung_Opacity/masks/Lung_Opacity-5697.png,True,G08621,test,266fdda28f743c053b35472f73ab0325,c42fc74b3da17132
Lung_Opacity-5698.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5698.png,Lung_Opacity/masks/Lung_Opacity-5698.png,True,G08622,test,e45bde433b21be70d5feea9f9026e22d,dc4f83050f0b1f5e
Lung_Opacity-5699.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5699.png,Lung_Opacity/masks/Lung_Opacity-5699.png,True,G08623,test,4478d9712190b9916c792f282655a215,996b871a1c475b59
Lung_Opacity-57.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-57.png,Lung_Opacity/masks/Lung_Opacity-57.png,True,G08624,train,8d614bd9faac5905bacdc787277a8bc9,e939171a1c4e5b55
Lung_Opacity-570.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-570.png,Lung_Opacity/masks/Lung_Opacity-570.png,True,G08625,train,2b22eb6f816db577b470b796f2b41053,d2281f5e1a7c7878
Lung_Opacity-5700.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5700.png,Lung_Opacity/masks/Lung_Opacity-5700.png,True,G08626,train,5b391fa316f0e3f5ef7a8fb924979f65,d03d5f081c0f6b9b
Lung_Opacity-5701.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5701.png,Lung_Opacity/masks/Lung_Opacity-5701.png,True,G08627,train,f2ce4d1ce7bebc2bcbde2dbf58d5140b,d52e0d5cea6aa8d1
Lung_Opacity-5702.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5702.png,Lung_Opacity/masks/Lung_Opacity-5702.png,True,G08628,train,ad2d64c3fee7002f3c2372613be3a798,9968c71ec21f1a79
Lung_Opacity-5703.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5703.png,Lung_Opacity/masks/Lung_Opacity-5703.png,True,G08629,train,54d23d660b37f7cc703166d1534e643f,903fc21a7a2b7b29
Lung_Opacity-5704.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5704.png,Lung_Opacity/masks/Lung_Opacity-5704.png,True,G08630,train,defb60a6f7047cb225f1f5a43d598db7,803b171f1f595d61
Lung_Opacity-5705.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5705.png,Lung_Opacity/masks/Lung_Opacity-5705.png,True,G08631,train,7cf08f8c66cfbd846f0b470bdce4a495,dc390b1af0f8d2c6
Lung_Opacity-5706.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5706.png,Lung_Opacity/masks/Lung_Opacity-5706.png,True,G08632,train,c8a30845967c82dcc84bbf7f93dce080,d42fc34d4d6c2d4c
Lung_Opacity-5707.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5707.png,Lung_Opacity/masks/Lung_Opacity-5707.png,True,G08633,train,e62ce1cfba5a5a9c9d504d3555b372a7,c13b760a9e0a1e7b
Lung_Opacity-5708.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5708.png,Lung_Opacity/masks/Lung_Opacity-5708.png,True,G08634,val,e446ea3caec59bb3dcdeda7a1700b80c,81781e5b5f686959
Lung_Opacity-5709.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5709.png,Lung_Opacity/masks/Lung_Opacity-5709.png,True,G08635,test,c7c7bad077e6d14a732cef1bffc96a62,dc6d4f0895d0d2e3
Lung_Opacity-571.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-571.png,Lung_Opacity/masks/Lung_Opacity-571.png,True,G08636,test,1a5dec7ec1417eaaaba6e9a1e2701f14,c46cc38b0d4f4b4f
Lung_Opacity-5710.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5710.png,Lung_Opacity/masks/Lung_Opacity-5710.png,True,G08637,train,bc2ff6d70e7f0a03f2c7354f792524d5,c13fd60a5e0b3f48
Lung_Opacity-5711.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5711.png,Lung_Opacity/masks/Lung_Opacity-5711.png,True,G08638,train,0f3683a86a7c2e7031da9b57435bc8fa,f96f861a0f1a0c5e
Lung_Opacity-5712.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5712.png,Lung_Opacity/masks/Lung_Opacity-5712.png,True,G08639,test,25cc48c5e3f6742a3ca5c8c17ed20933,817956c65b193c5e
Lung_Opacity-5713.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5713.png,Lung_Opacity/masks/Lung_Opacity-5713.png,True,G08640,train,b5e7b8da5542217508e1fb272ac21b67,cc2f021b491f4f4f
Lung_Opacity-5714.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5714.png,Lung_Opacity/masks/Lung_Opacity-5714.png,True,G08641,train,c1158dfa4ab9ccba45497ef8d26ceec8,956e4f5d482a686d
Lung_Opacity-5715.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5715.png,Lung_Opacity/masks/Lung_Opacity-5715.png,True,G08642,val,2be7ad18b386942b7be2d1cd7ac53e18,997987149e17339a
Lung_Opacity-5716.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5716.png,Lung_Opacity/masks/Lung_Opacity-5716.png,True,G08643,val,ec3ac202701ad1cb0597d71105afed8a,856f782363347e94
Lung_Opacity-5717.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5717.png,Lung_Opacity/masks/Lung_Opacity-5717.png,True,G08644,test,18c5d2bc7578dda87b9973081084584a,d46e074c4e6ec6e2
Lung_Opacity-5718.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5718.png,Lung_Opacity/masks/Lung_Opacity-5718.png,True,G08645,train,29004d3ce6dc5cf421a549243ea38fba,817a12185b7b7af2
Lung_Opacity-5719.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5719.png,Lung_Opacity/masks/Lung_Opacity-5719.png,True,G08646,train,27aecab5ec2c9b13e3dddae32f2c528c,d12dd6c34a9ccac9
Lung_Opacity-572.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-572.png,Lung_Opacity/masks/Lung_Opacity-572.png,True,G08647,train,099d5ca1cc49f0dfcd69471a496987bc,913b460d49636be7
Lung_Opacity-5720.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5720.png,Lung_Opacity/masks/Lung_Opacity-5720.png,True,G08648,train,d95ec440af4cf1516efe1cbd57bfa528,897c4e180d6d79e9
Lung_Opacity-5721.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5721.png,Lung_Opacity/masks/Lung_Opacity-5721.png,True,G08649,train,4ceed87d74f046e10dd20a37149a90b2,d95f961a2b386165
Lung_Opacity-5722.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5722.png,Lung_Opacity/masks/Lung_Opacity-5722.png,True,G08650,train,013fca03556489731a70f48a79a59732,cd2e875b4d4a6ce0
Lung_Opacity-5723.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5723.png,Lung_Opacity/masks/Lung_Opacity-5723.png,True,G08651,train,cd2fc6d58b647cdbe12d78286bda83b4,c939c6461ad16f3c
Lung_Opacity-5724.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5724.png,Lung_Opacity/masks/Lung_Opacity-5724.png,True,G08652,train,13740ddb3de52b634e7e4ed3385bf4f5,c12a164f6f2c6736
Lung_Opacity-5725.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5725.png,Lung_Opacity/masks/Lung_Opacity-5725.png,True,G08653,train,a88901a97889af4101b78e3c5bff749a,907d07c21ada3dce
Lung_Opacity-5726.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5726.png,Lung_Opacity/masks/Lung_Opacity-5726.png,True,G08654,train,af7c91e7256c12bd4bbfcca14e07ab88,942bc31d55595d59
Lung_Opacity-5727.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5727.png,Lung_Opacity/masks/Lung_Opacity-5727.png,True,G08655,train,f0f2702a0a8cb2c00bd7328548320ab7,d12bd6521d5d1b58
Lung_Opacity-5728.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5728.png,Lung_Opacity/masks/Lung_Opacity-5728.png,True,G08656,train,c2d2919d31305fcf644381591929f2b7,d46e03570b4f0a6f
Lung_Opacity-5729.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5729.png,Lung_Opacity/masks/Lung_Opacity-5729.png,True,G08657,train,cb69fc7dea87527cf4328c8c5bbaf273,d52ac34d5f480f2e
Lung_Opacity-573.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-573.png,Lung_Opacity/masks/Lung_Opacity-573.png,True,G08658,val,825f4dc763da82db6d42457a5d0dce98,8938155a5be9cae5
Lung_Opacity-5730.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5730.png,Lung_Opacity/masks/Lung_Opacity-5730.png,True,G08659,train,1ee022e2c2d669f91c2e8ee25748016c,d56e830b1d850dbd
Lung_Opacity-5731.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5731.png,Lung_Opacity/masks/Lung_Opacity-5731.png,True,G08660,test,a2cb1c45ae49a27991f344132eff436b,d42dd3534f6c704a
Lung_Opacity-5732.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5732.png,Lung_Opacity/masks/Lung_Opacity-5732.png,True,G08661,val,d57bbb2a003edcddd9dc218480d159dc,c03b121b5f2cbc75
Lung_Opacity-5733.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5733.png,Lung_Opacity/masks/Lung_Opacity-5733.png,True,G08662,test,dbcc4ec88ec109c1390f1bc87be2dd1c,cc3dd2431bc24b1f
Lung_Opacity-5734.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5734.png,Lung_Opacity/masks/Lung_Opacity-5734.png,True,G08663,test,6e2dad8218885dc0de80e69d1c9763af,dd0c474f4f0c0e4f
Lung_Opacity-5735.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5735.png,Lung_Opacity/masks/Lung_Opacity-5735.png,True,G08664,train,026994fb73a424864cdcb3a3cfe3750d,c87d865e545a5b4a
Lung_Opacity-5736.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5736.png,Lung_Opacity/masks/Lung_Opacity-5736.png,True,G08665,train,5bba83edbc7409cbdd4c0a987089e203,d03d5343df41db0c
Lung_Opacity-5737.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5737.png,Lung_Opacity/masks/Lung_Opacity-5737.png,True,G08666,test,3496380544436926596bf7d0edb74fcf,d03f435d580e4f1d
Lung_Opacity-5738.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5738.png,Lung_Opacity/masks/Lung_Opacity-5738.png,True,G08667,val,5ce7ddbfa0fc36f3d15de8537e991a26,916a1f1f5f606664
Lung_Opacity-5739.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5739.png,Lung_Opacity/masks/Lung_Opacity-5739.png,True,G08668,train,74762d05c30b6ff00e193003a8200204,c837d34656507d74
Lung_Opacity-574.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-574.png,Lung_Opacity/masks/Lung_Opacity-574.png,True,G08669,train,b1298edff9cc00b5e497a01626b4d9b8,dc3dc20b4b431f4b
Lung_Opacity-5740.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5740.png,Lung_Opacity/masks/Lung_Opacity-5740.png,True,G08670,train,d416993304c2be6583003c5121b9a553,8772517c786a6cac
Lung_Opacity-5741.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5741.png,Lung_Opacity/masks/Lung_Opacity-5741.png,True,G08671,train,c6f66304762a8ed88e1a51b3d87d2d1d,857c2e002c7f6d6d
Lung_Opacity-5742.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5742.png,Lung_Opacity/masks/Lung_Opacity-5742.png,True,G08672,train,5a3f9def212ccd17df945e6acc124336,902bc21f5c0f1f5e
Lung_Opacity-5743.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5743.png,Lung_Opacity/masks/Lung_Opacity-5743.png,True,G08673,train,c0c39aaeea33f822fcbd030df4dd4290,d96906de187909ed
Lung_Opacity-5744.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5744.png,Lung_Opacity/masks/Lung_Opacity-5744.png,True,G08674,train,3bcb3430caa64e3178b80c32f7666243,c13b46471e1e5a5e
Lung_Opacity-5745.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5745.png,Lung_Opacity/masks/Lung_Opacity-5745.png,True,G08675,train,9aae26c43e84fc5ad9c38392dd117b2a,d42f430b0e6e2f4e
Lung_Opacity-5746.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5746.png,Lung_Opacity/masks/Lung_Opacity-5746.png,True,G08676,train,84d9d76d9471f20450440894072a88c7,d42d0a2f3f333332
Lung_Opacity-5747.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5747.png,Lung_Opacity/masks/Lung_Opacity-5747.png,True,G08677,train,6116b61ada275e3cf6a57d12d03eaec8,897f1418194b5b5f
Lung_Opacity-5748.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5748.png,Lung_Opacity/masks/Lung_Opacity-5748.png,True,G08678,train,cd525aade6055191efd2a55b9c1782e1,d52ec74d0d4e4b43
Lung_Opacity-5749.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5749.png,Lung_Opacity/masks/Lung_Opacity-5749.png,True,G08679,train,0d0968b018df34aba794029426e14340,d62f521a2a3d6565
Lung_Opacity-575.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-575.png,Lung_Opacity/masks/Lung_Opacity-575.png,True,G08680,val,3a262364d63915c9fbb5702ddc0e90d3,89581f5b497969e8
Lung_Opacity-5750.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5750.png,Lung_Opacity/masks/Lung_Opacity-5750.png,True,G08681,val,4073e143bd5eb3c29c0d536989b9a033,d17f86180b393d39
Lung_Opacity-5751.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5751.png,Lung_Opacity/masks/Lung_Opacity-5751.png,True,G08682,val,ca3cbc3091a54ddcae7cf91ccaae690f,8978061e5b4b1f3d
Lung_Opacity-5752.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5752.png,Lung_Opacity/masks/Lung_Opacity-5752.png,True,G08683,val,65b850ee71b97e69b6a67882d6f3a167,917b5f465a1a584b
Lung_Opacity-5753.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5753.png,Lung_Opacity/masks/Lung_Opacity-5753.png,True,G08684,test,9fea2dea016531fd89d6fca389070c11,c9190a1f097f387d
Lung_Opacity-5754.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5754.png,Lung_Opacity/masks/Lung_Opacity-5754.png,True,G08685,test,2cd1e5df6c3265036be1957421116f8d,d03e8279864c7cfc
Lung_Opacity-5755.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5755.png,Lung_Opacity/masks/Lung_Opacity-5755.png,True,G08686,train,61c6c4966d3b6ccb013ea8f0e4809545,947ee1c11fc3071d
Lung_Opacity-5756.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5756.png,Lung_Opacity/masks/Lung_Opacity-5756.png,True,G08687,train,698f4f93e054738af89d26afe9c00dea,c03d6a1a5d2b5d2b
Lung_Opacity-5757.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5757.png,Lung_Opacity/masks/Lung_Opacity-5757.png,True,G08688,train,009ad557c19b646f7a78d65fa6034aea,996d4656494f0d4f
Lung_Opacity-5758.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5758.png,Lung_Opacity/masks/Lung_Opacity-5758.png,True,G08689,train,092f1862c72f9cb2dbf7aa6df69b7ad2,d96bd71c4cc2194e
Lung_Opacity-5759.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5759.png,Lung_Opacity/masks/Lung_Opacity-5759.png,True,G08690,train,ff672879bede793d179fff2feba75f65,9f3ae081171f5b4a
Lung_Opacity-576.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-576.png,Lung_Opacity/masks/Lung_Opacity-576.png,True,G08691,train,4601a983da869779b30a8ab2d29e2157,987905861a59ebbe
Lung_Opacity-5760.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5760.png,Lung_Opacity/masks/Lung_Opacity-5760.png,True,G08692,train,ba3279782a45d938366e7a1903b39d5a,dc3dc38b0a0b0f3d
Lung_Opacity-5761.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5761.png,Lung_Opacity/masks/Lung_Opacity-5761.png,True,G08693,train,538cb69db255c0bf65a2994257cbcf49,946e474b0d2e4c6f
Lung_Opacity-5762.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5762.png,Lung_Opacity/masks/Lung_Opacity-5762.png,True,G08694,train,280de704d163a0878136000a2168be10,827fc511500f5f3b
Lung_Opacity-5763.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5763.png,Lung_Opacity/masks/Lung_Opacity-5763.png,True,G08695,val,81d1f5abac2724cc273ae893f5885b58,996a471cce1dc82f
Lung_Opacity-5764.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5764.png,Lung_Opacity/masks/Lung_Opacity-5764.png,True,G08696,val,8cdd1815ed47646889c3802e9f8f3c58,8179171a197b7aa5
Lung_Opacity-5765.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5765.png,Lung_Opacity/masks/Lung_Opacity-5765.png,True,G08697,train,e76b6e3c68b8046b9fdc36074318b6d5,91395a1b1b397939
Lung_Opacity-5766.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5766.png,Lung_Opacity/masks/Lung_Opacity-5766.png,True,G08698,test,7ff57c53a7bc8f40d133b457684da306,d03b5f1a101dbf85
Lung_Opacity-5767.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5767.png,Lung_Opacity/masks/Lung_Opacity-5767.png,True,G08699,train,b2921e66de469fde376e022b19531555,9175841e55173737
Lung_Opacity-5768.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5768.png,Lung_Opacity/masks/Lung_Opacity-5768.png,True,G08700,train,f8934582d7578d28260a3cefb8c45e32,d80d0b0ebdf1f072
Lung_Opacity-5769.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5769.png,Lung_Opacity/masks/Lung_Opacity-5769.png,True,G08701,test,ef7ac9c3236c3c4caf87c6a7708b0c84,95780c5f5b4a5959
Lung_Opacity-577.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-577.png,Lung_Opacity/masks/Lung_Opacity-577.png,True,G08702,train,a49b5dddd01ca4358f0e95b1585f3aba,956884963f753879
Lung_Opacity-5770.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5770.png,Lung_Opacity/masks/Lung_Opacity-5770.png,True,G08703,train,aa2ea2077ed6797f73de73a2b01ac376,d03f52121e1b1f1f
Lung_Opacity-5771.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5771.png,Lung_Opacity/masks/Lung_Opacity-5771.png,True,G08704,test,cb978c5efd7ad738ab01b99625064306,d17f5a527b783012
Lung_Opacity-5772.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5772.png,Lung_Opacity/masks/Lung_Opacity-5772.png,True,G08705,train,906f839b24978ae2242c8b1731098fcd,993b4a0b4a28aebf
Lung_Opacity-5773.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5773.png,Lung_Opacity/masks/Lung_Opacity-5773.png,True,G08706,train,e335461143b22994ed28081a536a7cdd,916f07121d4f5c5d
Lung_Opacity-5774.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5774.png,Lung_Opacity/masks/Lung_Opacity-5774.png,True,G08707,train,9215f6d3022b14636d697d7e2d11b3cb,d0391a1b79e9a197
Lung_Opacity-5775.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5775.png,Lung_Opacity/masks/Lung_Opacity-5775.png,True,G08708,train,20974d2a680604749c549ff818edc1b1,d07d421a590b3f3e
Lung_Opacity-5776.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5776.png,Lung_Opacity/masks/Lung_Opacity-5776.png,True,G08709,train,84ad20f4c78c4fa240981bff1922fc56,d52acb0f6c2d3534
Lung_Opacity-5777.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5777.png,Lung_Opacity/masks/Lung_Opacity-5777.png,True,G08710,train,e6e2efb3f0b43e716c2f13584a6c3fa6,dd28d7531d58191d
Lung_Opacity-5778.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5778.png,Lung_Opacity/masks/Lung_Opacity-5778.png,True,G08711,train,df2aac1a49c40579e12826d97a3cde88,817f86165a1d595d
Lung_Opacity-5779.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5779.png,Lung_Opacity/masks/Lung_Opacity-5779.png,True,G08712,train,38f5679d3aaaf36357c8c1e9815fe9e3,cc7e161b1c0d0b6f
Lung_Opacity-578.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-578.png,Lung_Opacity/masks/Lung_Opacity-578.png,True,G08713,train,67eaee1b54d394759375005fe9ef1b64,817e161a596db8e9
Lung_Opacity-5780.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5780.png,Lung_Opacity/masks/Lung_Opacity-5780.png,True,G08714,train,7e06497b14ba276f30d0e8b4b5604faf,987f841e1a5f187a
Lung_Opacity-5781.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5781.png,Lung_Opacity/masks/Lung_Opacity-5781.png,True,G08715,train,0e916e71d79058e8c67f78e1fb3c6ee9,956c0e9f1d682b69
Lung_Opacity-5782.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5782.png,Lung_Opacity/masks/Lung_Opacity-5782.png,True,G08716,train,2f90d356680e7c25c13913d2409e46f9,dc2dc3530f2d2c4b
Lung_Opacity-5783.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5783.png,Lung_Opacity/masks/Lung_Opacity-5783.png,True,G08717,train,83cad10a26554be3c29980c7f4c44b79,913c46867a817bdd
Lung_Opacity-5784.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5784.png,Lung_Opacity/masks/Lung_Opacity-5784.png,True,G08718,val,d5bed79c51249627b2825aac08ce06f3,c36cb603960fd60f
Lung_Opacity-5785.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5785.png,Lung_Opacity/masks/Lung_Opacity-5785.png,True,G08719,val,6185d34b9df2eb7b70ecdf1e0b6deafc,c03d574f0f4c4e78
Lung_Opacity-5786.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5786.png,Lung_Opacity/masks/Lung_Opacity-5786.png,True,G08720,train,beb2f07d706a6e8bf258b51017419dab,d83b154e155a5d5a
Lung_Opacity-5787.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5787.png,Lung_Opacity/masks/Lung_Opacity-5787.png,True,G08721,val,bfa3c91f838a39bb93b3ae28032f073b,c93a130b7c68ede4
Lung_Opacity-5788.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5788.png,Lung_Opacity/masks/Lung_Opacity-5788.png,True,G08722,test,3ffe4c00dd55b8c45e43c23e824daf92,c82a1a4b0f2d7dad
Lung_Opacity-5789.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5789.png,Lung_Opacity/masks/Lung_Opacity-5789.png,True,G08723,test,c6b9655a97daac76027bc2bf29561823,d939d35a5c68383c
Lung_Opacity-579.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-579.png,Lung_Opacity/masks/Lung_Opacity-579.png,True,G08724,test,ecc6915ae40759e318cc5939507c7afe,95e1699d7a9d6818
Lung_Opacity-5790.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5790.png,Lung_Opacity/masks/Lung_Opacity-5790.png,True,G08725,train,8b4691b91fdfaca13a98ca02304c7652,d42f870d1d070f1f
Lung_Opacity-5791.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5791.png,Lung_Opacity/masks/Lung_Opacity-5791.png,True,G08726,train,8f4ac4111d636f1352312e96bdc663c0,817c06861f791d7d
Lung_Opacity-5792.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5792.png,Lung_Opacity/masks/Lung_Opacity-5792.png,True,G08727,train,5de320fc52f128f37d42a4d6b6f07b13,997e065a5d5869e8
Lung_Opacity-5793.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5793.png,Lung_Opacity/masks/Lung_Opacity-5793.png,True,G08728,train,f3537383ad7431ce22049e1d4bc93b9d,d83ba5060c175f1f
Lung_Opacity-5794.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5794.png,Lung_Opacity/masks/Lung_Opacity-5794.png,True,G08729,train,d83668e4404a40eaca88feffaf706d00,913f961a5d5a491e
Lung_Opacity-5795.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5795.png,Lung_Opacity/masks/Lung_Opacity-5795.png,True,G08730,train,ddfc860bfa6cf96e744ffe2be43e156d,95681e1f1949cc7e
Lung_Opacity-5796.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5796.png,Lung_Opacity/masks/Lung_Opacity-5796.png,True,G08731,train,f98ba966dc4e72f678ec4a87f2ec957f,d039530a0a29bfbf
Lung_Opacity-5797.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5797.png,Lung_Opacity/masks/Lung_Opacity-5797.png,True,G08732,train,ff683d4c534c09c014544a3f6b4becea,887b17871e187d8b
Lung_Opacity-5798.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5798.png,Lung_Opacity/masks/Lung_Opacity-5798.png,True,G08733,test,7c3ac46d78cef2efd89a5eeea9fc7a4b,d42e07570f4a6e6a
Lung_Opacity-5799.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5799.png,Lung_Opacity/masks/Lung_Opacity-5799.png,True,G08734,train,4f529658267a9644c8ee27fc4246f707,9d2ac30b1d4e4d5e
Lung_Opacity-58.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-58.png,Lung_Opacity/masks/Lung_Opacity-58.png,True,G08735,train,a3d225d2c88c5c7c78c38ba743606d6e,cc340b1f083f2e6f
Lung_Opacity-580.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-580.png,Lung_Opacity/masks/Lung_Opacity-580.png,True,G08736,val,8edb9dd1e87b78f944f45404676af7b3,9978061e185b5b5f
Lung_Opacity-5800.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5800.png,Lung_Opacity/masks/Lung_Opacity-5800.png,True,G08737,train,1e4a61f885d19e29cebf7a8c8309068c,896f86594dee60e2
Lung_Opacity-5801.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5801.png,Lung_Opacity/masks/Lung_Opacity-5801.png,True,G08738,test,59aefefaf6c0c00a1e3f77e2288e018c,c42f02704fb4ebf1
Lung_Opacity-5802.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5802.png,Lung_Opacity/masks/Lung_Opacity-5802.png,True,G08739,test,15b74cd57cbf3e19176542b923cbf8dc,c529141b4a2f7a3f
Lung_Opacity-5803.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5803.png,Lung_Opacity/masks/Lung_Opacity-5803.png,True,G08740,train,0cf2d2d7ec00b986b4cd2cc1b8511c52,d42ad31f18303f37
Lung_Opacity-5804.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5804.png,Lung_Opacity/masks/Lung_Opacity-5804.png,True,G08741,train,2f8cfd931ec95dc39ca657deadcb0149,9178581a5a5b3f2e
Lung_Opacity-5805.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5805.png,Lung_Opacity/masks/Lung_Opacity-5805.png,True,G08742,train,b14eb3a92df05c05c4053fc295ac5352,942d0e6c6a7a7971
Lung_Opacity-5806.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5806.png,Lung_Opacity/masks/Lung_Opacity-5806.png,True,G08743,val,13a39c0e54a66ff8d0bd59d675f27e15,c46f434f0e83abb2
Lung_Opacity-5807.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5807.png,Lung_Opacity/masks/Lung_Opacity-5807.png,True,G08744,val,e537413fae1d53b87cb8a7aadf7d99d7,d139161e161f1f1e
Lung_Opacity-5808.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5808.png,Lung_Opacity/masks/Lung_Opacity-5808.png,True,G08745,test,6dfc2fdb5199791f865a34b3f1d1c44a,912b5f9c7a916a98
Lung_Opacity-5809.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5809.png,Lung_Opacity/masks/Lung_Opacity-5809.png,True,G08746,train,a832cfc261006677cb8fa156cec3ffdd,d46e55c30f0e2e0f
Lung_Opacity-581.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-581.png,Lung_Opacity/masks/Lung_Opacity-581.png,True,G08747,train,c104356bc194e34faaa71be226dd336a,9139421f5b3d3939
Lung_Opacity-5810.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5810.png,Lung_Opacity/masks/Lung_Opacity-5810.png,True,G08748,val,a799f2118b89e0ca4670b8bbd282340c,8879061a5d5d4ede
Lung_Opacity-5811.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5811.png,Lung_Opacity/masks/Lung_Opacity-5811.png,True,G08749,val,b350527c4848f82667273c2cd13bd075,cc3d530f1a293e1d
Lung_Opacity-5812.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5812.png,Lung_Opacity/masks/Lung_Opacity-5812.png,True,G08750,train,2c800b75390f90c5abee45fd50bb4433,917a070b1e0e797e
Lung_Opacity-5813.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5813.png,Lung_Opacity/masks/Lung_Opacity-5813.png,True,G08751,train,67b25c9f9033436e1d1ebadbe9a74d7f,c03dd05b6a29697d
Lung_Opacity-5814.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5814.png,Lung_Opacity/masks/Lung_Opacity-5814.png,True,G08752,train,4a1f45ffebf7c4fbea2f95446ab5e12a,d47e43d35b194949
Lung_Opacity-5815.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5815.png,Lung_Opacity/masks/Lung_Opacity-5815.png,True,G08753,train,1cfcc50b53809791645905abd0396791,c54e074f49ece0f4
Lung_Opacity-5816.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5816.png,Lung_Opacity/masks/Lung_Opacity-5816.png,True,G08754,train,f77ff5c6e672dd979e1b247670db771a,823d5a1f5a3a38ba
Lung_Opacity-5817.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5817.png,Lung_Opacity/masks/Lung_Opacity-5817.png,True,G08755,train,099e59988f96db8a01002f54684069ec,d50a410f0e2d2fbf
Lung_Opacity-5818.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5818.png,Lung_Opacity/masks/Lung_Opacity-5818.png,True,G08756,train,ec386ea9fff40e292a97d46cc9ce4417,997a061a595b1b4f
Lung_Opacity-5819.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5819.png,Lung_Opacity/masks/Lung_Opacity-5819.png,True,G08757,train,97d11fd0f0af980ea782c9baf9b0f690,df78560a5c5b1919
Lung_Opacity-582.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-582.png,Lung_Opacity/masks/Lung_Opacity-582.png,True,G08758,train,d5a085eba3141a9e949dd5e4d6ea01ae,cd3fd213157a1d48
Lung_Opacity-5820.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5820.png,Lung_Opacity/masks/Lung_Opacity-5820.png,True,G08759,train,d469ddfef9cd0409bbe3a959fbbd07dc,833552575f7a1a4a
Lung_Opacity-5821.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5821.png,Lung_Opacity/masks/Lung_Opacity-5821.png,True,G08760,train,a2089ae5cf7417fbf8dab2b8dc2c8abe,8c4e4b2c443cc7df
Lung_Opacity-5822.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5822.png,Lung_Opacity/masks/Lung_Opacity-5822.png,True,G08761,train,fb702025b8578f7d5962aae55b6f32ae,d42d455d087f0b3d
Lung_Opacity-5823.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5823.png,Lung_Opacity/masks/Lung_Opacity-5823.png,True,G08762,val,8bf7deb270fd21eada216cd57a3b7bda,d03b530a2d293f3d
Lung_Opacity-5824.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5824.png,Lung_Opacity/masks/Lung_Opacity-5824.png,True,G08763,train,05da1366465b4be7cfab092b0478300b,c82c35525b1d2e5f
Lung_Opacity-5825.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5825.png,Lung_Opacity/masks/Lung_Opacity-5825.png,True,G08764,test,7dd9cacef934291f374f6af2f540f141,807d6f4e1a5d0d5c
Lung_Opacity-5826.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5826.png,Lung_Opacity/masks/Lung_Opacity-5826.png,True,G08765,test,70798821cf9df9e3ead913f3de349da7,917b435a1a5f1e1c
Lung_Opacity-5827.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5827.png,Lung_Opacity/masks/Lung_Opacity-5827.png,True,G08766,train,84459cf1599a45f683875e68519407ef,817f86158f057d1d
Lung_Opacity-5828.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5828.png,Lung_Opacity/masks/Lung_Opacity-5828.png,True,G08767,train,84b8ab8019590caef5c04f86756f7ece,c83b1506171b5f5b
Lung_Opacity-5829.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5829.png,Lung_Opacity/masks/Lung_Opacity-5829.png,True,G08768,train,3c7f86818a1944997670a635403892d5,803ece0e9f0d1f1d
Lung_Opacity-583.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-583.png,Lung_Opacity/masks/Lung_Opacity-583.png,True,G08769,train,52cbccdc01d091aeeeec43c2d08e1bd0,d42e024b1d6ceafc
Lung_Opacity-5830.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5830.png,Lung_Opacity/masks/Lung_Opacity-5830.png,True,G08770,train,9657ea8bf6594afe501fe3fb4c15ef76,8d680e175a6b6d78
Lung_Opacity-5831.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5831.png,Lung_Opacity/masks/Lung_Opacity-5831.png,True,G08771,train,d9ede625d38c21681eeccc38eed5dc1c,c12f141e095f1e7e
Lung_Opacity-5832.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5832.png,Lung_Opacity/masks/Lung_Opacity-5832.png,True,G08772,train,614ede0ff82933e80d59a3da1aecd525,c42f52d34b0fad4a
Lung_Opacity-5833.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5833.png,Lung_Opacity/masks/Lung_Opacity-5833.png,True,G08773,train,355373da2ebc9b0351f3f806199d4081,9978465a7b6027e6
Lung_Opacity-5834.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5834.png,Lung_Opacity/masks/Lung_Opacity-5834.png,True,G08774,train,dd09cb342abe5737493862ac9c736e13,946e6213ba0b5b5b
Lung_Opacity-5835.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5835.png,Lung_Opacity/masks/Lung_Opacity-5835.png,True,G08775,train,7b1abde42c9df1b93d0c85e35daec5aa,c03d6a6c5c7d0b5a
Lung_Opacity-5836.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5836.png,Lung_Opacity/masks/Lung_Opacity-5836.png,True,G08776,train,0a2ce73db0ae6c52060f5d97ab3e8d26,c03d0a1f1dbe9696
Lung_Opacity-5837.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5837.png,Lung_Opacity/masks/Lung_Opacity-5837.png,True,G08777,train,a174571933064a97f4d136cdcd8eec6c,817a181b1b5b5b5b
Lung_Opacity-5838.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5838.png,Lung_Opacity/masks/Lung_Opacity-5838.png,True,G08778,train,7ae23af4c35ca3d3eb3f76a6ed19b87d,d02df3f75e426818
Lung_Opacity-5839.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5839.png,Lung_Opacity/masks/Lung_Opacity-5839.png,True,G08779,train,6a60577d74b5fbec2838c719b4ed43b8,d42bc71c486f1acb
Lung_Opacity-584.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-584.png,Lung_Opacity/masks/Lung_Opacity-584.png,True,G08780,val,37a29b7c0bbe62292be05d5a6d4b7284,917847161e5f1a5b
Lung_Opacity-5840.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5840.png,Lung_Opacity/masks/Lung_Opacity-5840.png,True,G08781,val,d2fb06365d366a8ead795d6c86a5e8e3,dd2a574b4b4b0e0d
Lung_Opacity-5841.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5841.png,Lung_Opacity/masks/Lung_Opacity-5841.png,True,G08782,val,d98db098c26a105c0bf846519b2799a2,817fd606190d1f1f
Lung_Opacity-5842.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5842.png,Lung_Opacity/masks/Lung_Opacity-5842.png,True,G08783,test,c5143438db1ca041d9673587d262b282,dc3dd2494fc96889
Lung_Opacity-5843.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5843.png,Lung_Opacity/masks/Lung_Opacity-5843.png,True,G08784,train,6b1c831433f614cc3ebfdbe809335750,95684c5e494eadf8
Lung_Opacity-5844.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5844.png,Lung_Opacity/masks/Lung_Opacity-5844.png,True,G08785,test,db0e5139d0470b73eb3a9843740e4f61,9137541433133f3f
Lung_Opacity-5845.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5845.png,Lung_Opacity/masks/Lung_Opacity-5845.png,True,G08786,train,34fbcda0b7a84926751fa177dd5a2706,d92ad7561a59590d
Lung_Opacity-5846.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5846.png,Lung_Opacity/masks/Lung_Opacity-5846.png,True,G08787,test,d88b178ff625437297fb75a0f59146b4,d03d4b470f7cb918
Lung_Opacity-5847.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5847.png,Lung_Opacity/masks/Lung_Opacity-5847.png,True,G08788,train,1c80f65871e7c3877ad23a16df29e8f1,c43d1b4f1b4a6c6c
Lung_Opacity-5848.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5848.png,Lung_Opacity/masks/Lung_Opacity-5848.png,True,G08789,train,80c0ce6ba8698b74f4e8209b7be31107,dc2cd20b4d380def
Lung_Opacity-5849.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5849.png,Lung_Opacity/masks/Lung_Opacity-5849.png,True,G08790,train,6403f66a614db387fe0e8ff6d10ca803,995b541a193b3ba5
Lung_Opacity-585.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-585.png,Lung_Opacity/masks/Lung_Opacity-585.png,True,G08791,test,bd2a04d0344b24612bb319e6ee85ea96,857c460ffa0bfb08
Lung_Opacity-5850.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5850.png,Lung_Opacity/masks/Lung_Opacity-5850.png,True,G08792,train,8d2ba30991e24ad65fe19eb0ff9dc6d5,817bda58113b1b3b
Lung_Opacity-5851.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5851.png,Lung_Opacity/masks/Lung_Opacity-5851.png,True,G08793,train,3d3bdc8ea92c9e3688883ec0f0aa635a,d52b424f0c0c7cbf
Lung_Opacity-5852.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5852.png,Lung_Opacity/masks/Lung_Opacity-5852.png,True,G08794,train,15d069640c44b7c2fcf19ed0aad47726,d42ad3cb4d0f0e4e
Lung_Opacity-5853.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5853.png,Lung_Opacity/masks/Lung_Opacity-5853.png,True,G08795,train,323a8c83ba97cef1c77d74810da3dee7,817b961a7819793d
Lung_Opacity-5854.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5854.png,Lung_Opacity/masks/Lung_Opacity-5854.png,True,G08796,train,1e48970d292ded05eb673b0b5c81052e,dd28861e1d493d4f
Lung_Opacity-5855.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5855.png,Lung_Opacity/masks/Lung_Opacity-5855.png,True,G08797,train,74908abd5933a5eb4c119131c1455f07,856e0e160f4f5d1e
Lung_Opacity-5856.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5856.png,Lung_Opacity/masks/Lung_Opacity-5856.png,True,G08798,val,7cb6682c6eb8dd2b62c2724c87c1e415,d42b431b0a3d3cb7
Lung_Opacity-5857.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5857.png,Lung_Opacity/masks/Lung_Opacity-5857.png,True,G08799,train,a50cf0058faa7273f5be89dfe7a564c6,837a54105b5b7b5a
Lung_Opacity-5858.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5858.png,Lung_Opacity/masks/Lung_Opacity-5858.png,True,G08800,train,dde5b736c3ef70959b51e5d0da181f2e,913d6286e599599b
Lung_Opacity-5859.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5859.png,Lung_Opacity/masks/Lung_Opacity-5859.png,True,G08801,train,8812f479dd79bd898d19af45ee6ac235,d52fbe078f0d0c0d
Lung_Opacity-586.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-586.png,Lung_Opacity/masks/Lung_Opacity-586.png,True,G08802,val,71291aaf0631ec7d88faad9412c64426,897e96195d1b1947
Lung_Opacity-5860.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5860.png,Lung_Opacity/masks/Lung_Opacity-5860.png,True,G08803,val,df79e24c2c7aa4bebba41dce91b374e8,89687c945a4d1f5b
Lung_Opacity-5861.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5861.png,Lung_Opacity/masks/Lung_Opacity-5861.png,True,G08804,test,12f6a60f31dcf3b65a9e4d10ce851073,913a1c5f794acec2
Lung_Opacity-5862.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5862.png,Lung_Opacity/masks/Lung_Opacity-5862.png,True,G08805,test,59f000cbf3e51f6e291faecebf8c6ffc,897a871a145979e7
Lung_Opacity-5863.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5863.png,Lung_Opacity/masks/Lung_Opacity-5863.png,True,G08806,train,bde27ecc32530e2970253d478ec48586,d97e171b19493869
Lung_Opacity-5864.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5864.png,Lung_Opacity/masks/Lung_Opacity-5864.png,True,G08807,train,d7e8ad5d25b3727512535156935d6052,813fd480d00fd9df
Lung_Opacity-5865.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5865.png,Lung_Opacity/masks/Lung_Opacity-5865.png,True,G08808,train,268a285e3783352a884a77a77588cf40,dc2e4d0c2e4afad2
Lung_Opacity-5866.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5866.png,Lung_Opacity/masks/Lung_Opacity-5866.png,True,G08809,train,5d6b12ad8a0be2f339079d219564a43d,dd2ad35f560e4c12
Lung_Opacity-5867.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5867.png,Lung_Opacity/masks/Lung_Opacity-5867.png,True,G08810,train,b1eb7d9f198aa3924ba232ca7f4704eb,c42f472fb1333139
Lung_Opacity-5868.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5868.png,Lung_Opacity/masks/Lung_Opacity-5868.png,True,G08811,train,a2fe5dffd15d6e0f178bd0520c9ff47a,8760456b0f4f4f4b
Lung_Opacity-5869.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5869.png,Lung_Opacity/masks/Lung_Opacity-5869.png,True,G08812,train,e6790b62500177165a4baa5855ae6f65,91684f1b4f5c5a4e
Lung_Opacity-587.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-587.png,Lung_Opacity/masks/Lung_Opacity-587.png,True,G08813,train,a827cc884a7cd67f057372112a73b497,917c861e0b0f1f5e
Lung_Opacity-5870.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5870.png,Lung_Opacity/masks/Lung_Opacity-5870.png,True,G08814,train,4a27ea0046c9fa272f76e60107417aad,9931c2da5b4f4b49
Lung_Opacity-5871.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5871.png,Lung_Opacity/masks/Lung_Opacity-5871.png,True,G08815,val,1c48c614ab5eb9c89624da41f0efd4ce,996a471e1a1b1b2f
Lung_Opacity-5872.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5872.png,Lung_Opacity/masks/Lung_Opacity-5872.png,True,G08816,train,90671a53cbd122e1e5efede2cebce471,d13f52044b4b5b5b
Lung_Opacity-5873.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5873.png,Lung_Opacity/masks/Lung_Opacity-5873.png,True,G08817,train,2fecd47d48f1e0b3922d4e9612c51787,dd2c43030f5b6c3d
Lung_Opacity-5874.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5874.png,Lung_Opacity/masks/Lung_Opacity-5874.png,True,G08818,train,4277cccf2767c7dc32688b98a6cf42cb,903a161a1ebabfb8
Lung_Opacity-5875.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5875.png,Lung_Opacity/masks/Lung_Opacity-5875.png,True,G08819,test,cf7a395a89acf32a8317175c7f0457c9,9c6a410d0f4e2f7b
Lung_Opacity-5876.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5876.png,Lung_Opacity/masks/Lung_Opacity-5876.png,True,G08820,train,cc5cd7298486b1124cbcaeedda103447,dc2d070e4e6e6ce4
Lung_Opacity-5877.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5877.png,Lung_Opacity/masks/Lung_Opacity-5877.png,True,G08821,val,27f10bc86658f0be5475e44f0bdbd933,dc54424b0b4b6d7d
Lung_Opacity-5878.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5878.png,Lung_Opacity/masks/Lung_Opacity-5878.png,True,G08822,train,df58429ccd897ea4b5e17896ace1b93c,807bc0c859abfeb4
Lung_Opacity-5879.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5879.png,Lung_Opacity/masks/Lung_Opacity-5879.png,True,G08823,train,817a78c9065985a551d763d2b12e629d,817a450a1b1f5e5f
Lung_Opacity-588.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-588.png,Lung_Opacity/masks/Lung_Opacity-588.png,True,G08824,val,1b7889aa2fedc054e42b677f9cecfa9c,897b861e184f7b31
Lung_Opacity-5880.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5880.png,Lung_Opacity/masks/Lung_Opacity-5880.png,True,G08825,test,cb2e1ebf355be0d2321a6ce1d999fa34,d93ad20b0d191f3d
Lung_Opacity-5881.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5881.png,Lung_Opacity/masks/Lung_Opacity-5881.png,True,G08826,test,f77094d9a0f5506148239d975ecd0b0d,95780c0f4a6bc7c7
Lung_Opacity-5882.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5882.png,Lung_Opacity/masks/Lung_Opacity-5882.png,True,G08827,test,a0ecd5dc0297d7c985e1d994bffb4162,c87f535b1c4b1d18
Lung_Opacity-5883.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5883.png,Lung_Opacity/masks/Lung_Opacity-5883.png,True,G08828,train,a50238a39906520dcb9830efed75cb19,d427d1c94e626b0f
Lung_Opacity-5884.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5884.png,Lung_Opacity/masks/Lung_Opacity-5884.png,True,G08829,test,44581bc99557c7e99500fb8f10680b9c,d42f0d4c48ebe9f0
Lung_Opacity-5885.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5885.png,Lung_Opacity/masks/Lung_Opacity-5885.png,True,G08830,test,e102177873602cfa9ba5bba4be130d30,903f06120d5f5f4f
Lung_Opacity-5886.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5886.png,Lung_Opacity/masks/Lung_Opacity-5886.png,True,G08831,train,c06bcecccba354f16e8733e8bfdf365f,812be5849715d74f
Lung_Opacity-5887.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5887.png,Lung_Opacity/masks/Lung_Opacity-5887.png,True,G08832,train,3ce5182d7d689c9e0a54e7ef3e0105b1,9f7a157c135858d8
Lung_Opacity-5888.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5888.png,Lung_Opacity/masks/Lung_Opacity-5888.png,True,G08833,test,3e100c01a3df02d6db71e2859d8ceb49,9168041e165e7b7f
Lung_Opacity-5889.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5889.png,Lung_Opacity/masks/Lung_Opacity-5889.png,True,G08834,train,bd5ebabd86b934b1c0efa60c28745ebe,d87f5c18185a5a79
Lung_Opacity-589.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-589.png,Lung_Opacity/masks/Lung_Opacity-589.png,True,G08835,train,153cdb83334365a1a6fe0ea2aca7c644,9d280f1e5e4c395b
Lung_Opacity-5890.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5890.png,Lung_Opacity/masks/Lung_Opacity-5890.png,True,G08836,train,a3812af7a148d7b28553df63dc218d71,8178155a5f6a5b3a
Lung_Opacity-5891.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5891.png,Lung_Opacity/masks/Lung_Opacity-5891.png,True,G08837,train,6ff7a311acb7f8779979bc52bb37cb9e,917b47921ac93abc
Lung_Opacity-5892.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5892.png,Lung_Opacity/masks/Lung_Opacity-5892.png,True,G08838,train,c829485c6412fe25458c93b37b38ce59,892a0f5e7ceae8e0
Lung_Opacity-5893.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5893.png,Lung_Opacity/masks/Lung_Opacity-5893.png,True,G08839,train,08d2935d355e66fb2d451c5a003c1eaa,c1ac3a031d5f3bc9
Lung_Opacity-5894.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5894.png,Lung_Opacity/masks/Lung_Opacity-5894.png,True,G08840,train,f3acb3d549fff8684cc604befdb6ed63,d8185b0a3c7d71e3
Lung_Opacity-5895.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5895.png,Lung_Opacity/masks/Lung_Opacity-5895.png,True,G08841,train,9aa876622ee63a858eb37256123a12fb,9f680f07161e1f1e
Lung_Opacity-5896.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5896.png,Lung_Opacity/masks/Lung_Opacity-5896.png,True,G08842,train,3115bfe9bfa331ccccaa4861f040004a,91688e1f9f34315d
Lung_Opacity-5897.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5897.png,Lung_Opacity/masks/Lung_Opacity-5897.png,True,G08843,train,fa80ac0c91758e4ce40e115100ac849a,807b541e5a1b3b3b
Lung_Opacity-5898.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5898.png,Lung_Opacity/masks/Lung_Opacity-5898.png,True,G08844,val,8689005fa4cf9c4984114705e2533807,8b78167f6c646465
Lung_Opacity-5899.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5899.png,Lung_Opacity/masks/Lung_Opacity-5899.png,True,G08845,val,363c0116bb0e0487b2c069d695b3e84d,d0297b795855591b
Lung_Opacity-59.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-59.png,Lung_Opacity/masks/Lung_Opacity-59.png,True,G08846,val,dbee9795341a8aa59623e209f7af8401,817942121b6f3f3b
Lung_Opacity-590.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-590.png,Lung_Opacity/masks/Lung_Opacity-590.png,True,G08847,train,dbce9e09e86a291ca2c96b5c60f7d871,d42ed3590e0e3f0e
Lung_Opacity-5900.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5900.png,Lung_Opacity/masks/Lung_Opacity-5900.png,True,G08848,train,5fab34a91d8eab7be22eeab8516ad4dc,907f47cf16194b15
Lung_Opacity-5901.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5901.png,Lung_Opacity/masks/Lung_Opacity-5901.png,True,G08849,train,d1f16cafdb8d2245f8baae5216d8d99c,d42fd341df081c5e
Lung_Opacity-5902.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5902.png,Lung_Opacity/masks/Lung_Opacity-5902.png,True,G08850,train,2e6a9123e642cd3877817174e511b6ea,d52a474f4a1e5c5c
Lung_Opacity-5903.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5903.png,Lung_Opacity/masks/Lung_Opacity-5903.png,True,G08851,train,b1711dcfe5f910754e7a01edca6cafa6,dd4fc30b6ca0b1b2
Lung_Opacity-5904.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5904.png,Lung_Opacity/masks/Lung_Opacity-5904.png,True,G08852,train,3dfc470e88664fbe66bb31e65c32b24c,836a343512523f7f
Lung_Opacity-5905.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5905.png,Lung_Opacity/masks/Lung_Opacity-5905.png,True,G08853,train,76b240300f28b88c45986f3dc7a3f63e,847ec7495d6d0d4c
Lung_Opacity-5906.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5906.png,Lung_Opacity/masks/Lung_Opacity-5906.png,True,G08854,train,ca7a8a3f76598e3aaaa4ee5107ae0bac,c625434e661bf1f1
Lung_Opacity-5907.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5907.png,Lung_Opacity/masks/Lung_Opacity-5907.png,True,G08855,train,1b783342f91eb4e7ede9c77000a00a56,c23d0e2b71733333
Lung_Opacity-5908.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5908.png,Lung_Opacity/masks/Lung_Opacity-5908.png,True,G08856,train,50fc3601d17ffddb20354ad397eb8f74,c07bd7571e121a39
Lung_Opacity-5909.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5909.png,Lung_Opacity/masks/Lung_Opacity-5909.png,True,G08857,train,2a1f047f613b8df956322a5afebfcf48,c52fc64f5a68e8d0
Lung_Opacity-591.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-591.png,Lung_Opacity/masks/Lung_Opacity-591.png,True,G08858,train,ee769eb55a5d07359ed09b4dbc9d3402,9d68471a590f1e4f
Lung_Opacity-5910.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5910.png,Lung_Opacity/masks/Lung_Opacity-5910.png,True,G08859,train,d999396d1830e3d862cd8d0418389f79,cd2a570a5d4b5a5c
Lung_Opacity-5911.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5911.png,Lung_Opacity/masks/Lung_Opacity-5911.png,True,G08860,train,7f274a0f1e80290c09f21655970df071,d83b561e55191d3c
Lung_Opacity-5912.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5912.png,Lung_Opacity/masks/Lung_Opacity-5912.png,True,G08861,val,de30bcdbf8d04a7b953c980c6e385984,9332d4145e5a7b3a
Lung_Opacity-5913.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5913.png,Lung_Opacity/masks/Lung_Opacity-5913.png,True,G08862,val,8a388779f0f6b72283799ad99322e8df,817cc61a581f5e5e
Lung_Opacity-5914.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5914.png,Lung_Opacity/masks/Lung_Opacity-5914.png,True,G08863,val,b56bd9f821440c72f9e4cca2ec2c187a,dc39430f1c2c3db5
Lung_Opacity-5915.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5915.png,Lung_Opacity/masks/Lung_Opacity-5915.png,True,G08864,train,f4d67e9dbfc645d6e83071efe61234e3,9818139bf8e2f6e2
Lung_Opacity-5916.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5916.png,Lung_Opacity/masks/Lung_Opacity-5916.png,True,G08865,test,328aa1b6430de1aa8b035d87eac20e91,9969c61e525d78e8
Lung_Opacity-5917.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5917.png,Lung_Opacity/masks/Lung_Opacity-5917.png,True,G08866,test,d0920310444f9c44c96e46faae2d79bc,997886167a097d5d
Lung_Opacity-5918.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5918.png,Lung_Opacity/masks/Lung_Opacity-5918.png,True,G08867,train,0703422bfbb706cb7109e6483b11677e,9f701d1f525253d8
Lung_Opacity-5919.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5919.png,Lung_Opacity/masks/Lung_Opacity-5919.png,True,G08868,train,ad6907f0b15d1b54880914eb6f6dea49,d12f561e1e0e4f0d
Lung_Opacity-592.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-592.png,Lung_Opacity/masks/Lung_Opacity-592.png,True,G08869,train,f068cf79a55f6655da6620cefe6b14a8,d13b5e085b3b182f
Lung_Opacity-5920.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5920.png,Lung_Opacity/masks/Lung_Opacity-5920.png,True,G08870,train,3f67fdbecacc01095aff94d69d5eb519,813f461a185e7f59
Lung_Opacity-5921.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5921.png,Lung_Opacity/masks/Lung_Opacity-5921.png,True,G08871,train,82c282e590500588a439dac81247f302,c42fe5470f444e4f
Lung_Opacity-5922.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5922.png,Lung_Opacity/masks/Lung_Opacity-5922.png,True,G08872,train,d599d94162fd0d86aa4161d77189d711,d02d4b081e3e3d3f
Lung_Opacity-5923.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5923.png,Lung_Opacity/masks/Lung_Opacity-5923.png,True,G08873,train,629e3966c5e963b25129d09f924b16f4,c13f86960f451d1f
Lung_Opacity-5924.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5924.png,Lung_Opacity/masks/Lung_Opacity-5924.png,True,G08874,train,ad68b7a88048ae51164bead023c065be,95681e5e786969e1
Lung_Opacity-5925.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5925.png,Lung_Opacity/masks/Lung_Opacity-5925.png,True,G08875,val,e8185a2eee0f62d596917f1275d585e4,c03fda1c1d3d3c1a
Lung_Opacity-5926.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5926.png,Lung_Opacity/masks/Lung_Opacity-5926.png,True,G08876,train,2b2dcaa372deef7a40eb2ddac470bf5a,dc2a5d0c2ca8bdb5
Lung_Opacity-5927.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5927.png,Lung_Opacity/masks/Lung_Opacity-5927.png,True,G08877,train,3a95041abcf85ca87e133c89a325feaa,d53951c2574beb41
Lung_Opacity-5928.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5928.png,Lung_Opacity/masks/Lung_Opacity-5928.png,True,G08878,train,5574604ba95f91ef0f47542bf4ca286a,d42f5a4a4e4e4e4e
Lung_Opacity-5929.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5929.png,Lung_Opacity/masks/Lung_Opacity-5929.png,True,G08879,train,b9ddb2061e68a2a6e2f7487cf8529a82,817bd61e5d09251f
Lung_Opacity-593.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-593.png,Lung_Opacity/masks/Lung_Opacity-593.png,True,G08880,train,7994400b36131b8726c794cd59babc85,dc3893db0d3c928b
Lung_Opacity-5930.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5930.png,Lung_Opacity/masks/Lung_Opacity-5930.png,True,G08881,train,1d39f03955c2bb5c5fabb26ff2a648b4,c17e1e1c194d0daf
Lung_Opacity-5931.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5931.png,Lung_Opacity/masks/Lung_Opacity-5931.png,True,G08882,val,0f880d9349af82010871c2279abdb709,89681d861e5c5f5b
Lung_Opacity-5932.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5932.png,Lung_Opacity/masks/Lung_Opacity-5932.png,True,G08883,test,a41e717644a87e78b8d9050497429189,9c6dc71e1e491d4c
Lung_Opacity-5933.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5933.png,Lung_Opacity/masks/Lung_Opacity-5933.png,True,G08884,test,a5642779cd45b19f0cf3854928d4de96,9978879686965e17
Lung_Opacity-5934.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5934.png,Lung_Opacity/masks/Lung_Opacity-5934.png,True,G08885,train,370f14822f424a7d36f3ec4e352f02e4,943e43470f0d7d1e
Lung_Opacity-5935.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5935.png,Lung_Opacity/masks/Lung_Opacity-5935.png,True,G08886,val,8142535046d53fd53dacffc900d20916,8969161a075e5d5f
Lung_Opacity-5936.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5936.png,Lung_Opacity/masks/Lung_Opacity-5936.png,True,G08887,test,452030d3a0c874846eba5328f18b806e,dc0a931b1c3a2df9
Lung_Opacity-5937.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5937.png,Lung_Opacity/masks/Lung_Opacity-5937.png,True,G08888,train,c9ba3bbbfe9b0c75d88f5a0e23f978c0,943e03d31b4f0f0f
Lung_Opacity-5938.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5938.png,Lung_Opacity/masks/Lung_Opacity-5938.png,True,G08889,train,e4ab18f6b0123e9a6c59a1127ff427e5,dd2ad20b155a5b1e
Lung_Opacity-5939.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5939.png,Lung_Opacity/masks/Lung_Opacity-5939.png,True,G08890,train,2671ba4e68485c5d86da79c55b582555,c12bb61e5a1d5e1a
Lung_Opacity-594.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-594.png,Lung_Opacity/masks/Lung_Opacity-594.png,True,G08891,train,ca70ea34b83e454adf2166d05f00d436,d52947065f0b474f
Lung_Opacity-5940.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5940.png,Lung_Opacity/masks/Lung_Opacity-5940.png,True,G08892,val,7d6daaba0fdf688b45d07ce2b159a26d,d42e934f071f2d07
Lung_Opacity-5941.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5941.png,Lung_Opacity/masks/Lung_Opacity-5941.png,True,G08893,train,57f06036554fa48fe632149e0dda9df1,817f025e165f591b
Lung_Opacity-5942.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5942.png,Lung_Opacity/masks/Lung_Opacity-5942.png,True,G08894,test,8066f0d0469c28ca29aacf96928a9eaa,8c0bc30b4d4f5f4b
Lung_Opacity-5943.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5943.png,Lung_Opacity/masks/Lung_Opacity-5943.png,True,G08895,train,cfc95cafecac1d9466f20c93350c954c,d42b474f4f0e0e1e
Lung_Opacity-5944.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5944.png,Lung_Opacity/masks/Lung_Opacity-5944.png,True,G08896,train,d9194017d0eacbf2bb26daf954ccf278,d42c6f41db594f0a
Lung_Opacity-5945.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5945.png,Lung_Opacity/masks/Lung_Opacity-5945.png,True,G08897,train,2e8756bb619834e96d5a9bfb2e77a26f,dc2b431b591d1c4f
Lung_Opacity-5946.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5946.png,Lung_Opacity/masks/Lung_Opacity-5946.png,True,G08898,train,f06db15b67cb5ec6422e8db0162baa76,996bc71cca38495d
Lung_Opacity-5947.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5947.png,Lung_Opacity/masks/Lung_Opacity-5947.png,True,G08899,train,f1f28645503e1e2870dc56971f6f900d,957f8e155d05194d
Lung_Opacity-5948.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5948.png,Lung_Opacity/masks/Lung_Opacity-5948.png,True,G08900,train,a3198fb9d42dea98ebb4064d6fe93c70,8330849fb9b954dd
Lung_Opacity-5949.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5949.png,Lung_Opacity/masks/Lung_Opacity-5949.png,True,G08901,train,4f930a1dc12400d98acb25bc7f04ff75,992c075b4f5e4c4e
Lung_Opacity-595.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-595.png,Lung_Opacity/masks/Lung_Opacity-595.png,True,G08902,val,d6e505adffd65ec975e43c75423ca7b8,8a3dc243df1a4e4e
Lung_Opacity-5950.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5950.png,Lung_Opacity/masks/Lung_Opacity-5950.png,True,G08903,val,bf28c4c2b55924db7e691c4ce1737b0a,85781d1d5b5d5a58
Lung_Opacity-5951.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5951.png,Lung_Opacity/masks/Lung_Opacity-5951.png,True,G08904,test,923f85199eecd67a5730782693220dd9,d43d42135fc34357
Lung_Opacity-5952.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5952.png,Lung_Opacity/masks/Lung_Opacity-5952.png,True,G08905,train,a32e6f4a315e96a7c07a0d975515a62b,8d6c1e5a494c6d6d
Lung_Opacity-5953.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5953.png,Lung_Opacity/masks/Lung_Opacity-5953.png,True,G08906,train,f7a81e71d364af714ccd3068535bf89d,897b8752495f584b
Lung_Opacity-5954.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5954.png,Lung_Opacity/masks/Lung_Opacity-5954.png,True,G08907,test,aa03007d832012b19a6c540ccc64e178,84611f1f1f737161
Lung_Opacity-5955.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5955.png,Lung_Opacity/masks/Lung_Opacity-5955.png,True,G08908,train,a4bda4413633c01e5af80e866832a083,856e064f45474fc7
Lung_Opacity-5956.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5956.png,Lung_Opacity/masks/Lung_Opacity-5956.png,True,G08909,train,354ddfda9ed09cd362db6fb97059d26f,cc3f3253544c5b4d
Lung_Opacity-5957.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5957.png,Lung_Opacity/masks/Lung_Opacity-5957.png,True,G08910,train,f7021e18d137c2bc45c4cbc99c87c2b3,9929061c185f79ef
Lung_Opacity-5958.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5958.png,Lung_Opacity/masks/Lung_Opacity-5958.png,True,G08911,train,e3e78d4dd9e6dd5476d90b518226c187,c52f96101f1f1f0d
Lung_Opacity-5959.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5959.png,Lung_Opacity/masks/Lung_Opacity-5959.png,True,G08912,train,7b9e762feafedae880ad6a6c6745737c,d62f0a3d79343233
Lung_Opacity-596.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-596.png,Lung_Opacity/masks/Lung_Opacity-596.png,True,G08913,train,00001a8511fe190756f9e6744de2de11,917b57861a992b99
Lung_Opacity-5960.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5960.png,Lung_Opacity/masks/Lung_Opacity-5960.png,True,G08914,train,2142f8d61e160839a0efbfb46c36a972,d52e820f4c7c69ce
Lung_Opacity-5961.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5961.png,Lung_Opacity/masks/Lung_Opacity-5961.png,True,G08915,train,7037c721ceaf69280d9db821586cb5ed,c8356b424b0bed6d
Lung_Opacity-5962.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5962.png,Lung_Opacity/masks/Lung_Opacity-5962.png,True,G08916,train,96216a557cbda0008b9128c8c7dbdbd4,d62dd94547242f6c
Lung_Opacity-5963.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5963.png,Lung_Opacity/masks/Lung_Opacity-5963.png,True,G08917,train,32470f8decaffd5519341392306b9827,817b4d8c5179e8cb
Lung_Opacity-5964.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5964.png,Lung_Opacity/masks/Lung_Opacity-5964.png,True,G08918,train,8693dc5cd2550049eec9e390a893f988,917a431f3d182e3e
Lung_Opacity-5965.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5965.png,Lung_Opacity/masks/Lung_Opacity-5965.png,True,G08919,train,13ba741faed0fc0fafbb996d276e2694,c43e5e4e082dece6
Lung_Opacity-5966.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5966.png,Lung_Opacity/masks/Lung_Opacity-5966.png,True,G08920,test,1617a1e2e00962fae66efd7686256fe8,915a1e5b79e1a0a7
Lung_Opacity-5967.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5967.png,Lung_Opacity/masks/Lung_Opacity-5967.png,True,G08921,train,ebc66bd2ebc5fd0e9878ddb579bf967f,d03d594a4a4a6f6b
Lung_Opacity-5968.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5968.png,Lung_Opacity/masks/Lung_Opacity-5968.png,True,G08922,train,7f6250dd43be1ee940dbc7aa9369446a,85788505fd178b57
Lung_Opacity-5969.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5969.png,Lung_Opacity/masks/Lung_Opacity-5969.png,True,G08923,val,d21aa0de3cd18079008aa5b2204768bb,c96fc259f8e2b290
Lung_Opacity-597.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-597.png,Lung_Opacity/masks/Lung_Opacity-597.png,True,G08924,val,f3faf876e8a16e25e7cecc5f145bb6d8,dc2dc70287071f1f
Lung_Opacity-5970.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5970.png,Lung_Opacity/masks/Lung_Opacity-5970.png,True,G08925,val,5756c81e20501413f7c7e099bbf45d7c,dc2ce0931b494d3f
Lung_Opacity-5971.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5971.png,Lung_Opacity/masks/Lung_Opacity-5971.png,True,G08926,test,bf0e8ab93c330570e530b9ffaf727e3b,906ccf1f58453d5c
Lung_Opacity-5972.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5972.png,Lung_Opacity/masks/Lung_Opacity-5972.png,True,G08927,test,0162e6eaf5acb29f545cb7054b33da2b,d62d554a0f4f4c78
Lung_Opacity-5973.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5973.png,Lung_Opacity/masks/Lung_Opacity-5973.png,True,G08928,train,66b3a3260fbf96dbe21b780db0f6ac4e,c425cb4e4a0b6fe3
Lung_Opacity-5974.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5974.png,Lung_Opacity/masks/Lung_Opacity-5974.png,True,G08929,train,97de29da7fcc2330695926288b7e8eac,c02d4b4b0b0faf2f
Lung_Opacity-5975.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5975.png,Lung_Opacity/masks/Lung_Opacity-5975.png,True,G08930,train,8a488f92f35900dde3f98faf867481e7,d57a075d090f0f0f
Lung_Opacity-5976.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5976.png,Lung_Opacity/masks/Lung_Opacity-5976.png,True,G08931,train,af0a9cacf037693d5da62b33d67334fb,9168061e1b1f3dbd
Lung_Opacity-5977.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5977.png,Lung_Opacity/masks/Lung_Opacity-5977.png,True,G08932,train,9436b5741e7482777af6c5c18a6f4e39,c52f16034f0f0f4f
Lung_Opacity-5978.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5978.png,Lung_Opacity/masks/Lung_Opacity-5978.png,True,G08933,train,8b514c6e3dcdc2239adfa410f47cb02e,817cc71a1e571b59
Lung_Opacity-5979.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5979.png,Lung_Opacity/masks/Lung_Opacity-5979.png,True,G08934,train,80268c0aa48ea75bea91a3f984cad1f8,817816111b3b7b3f
Lung_Opacity-598.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-598.png,Lung_Opacity/masks/Lung_Opacity-598.png,True,G08935,train,038dda2f9765d14de7ba19065991654e,cd6f1603531f4c5a
Lung_Opacity-5980.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5980.png,Lung_Opacity/masks/Lung_Opacity-5980.png,True,G08936,train,b2e2f8da6730e6586c049d533b818849,982d87968f071f0f
Lung_Opacity-5981.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5981.png,Lung_Opacity/masks/Lung_Opacity-5981.png,True,G08937,train,dbe6f17e48bf885885115d843866ba94,d129425f5e0b5f19
Lung_Opacity-5982.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5982.png,Lung_Opacity/masks/Lung_Opacity-5982.png,True,G08938,train,090352493b6f571f1aad5527168674e0,c8dd4bca3d883d94
Lung_Opacity-5983.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5983.png,Lung_Opacity/masks/Lung_Opacity-5983.png,True,G08939,train,c2199860a7eb870ba1fbc9db9bb9f076,9919461e1c5f1a7b
Lung_Opacity-5984.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5984.png,Lung_Opacity/masks/Lung_Opacity-5984.png,True,G08940,train,bc75264702c41b951e504244252d3549,9d59c24f491c5d4b
Lung_Opacity-5985.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5985.png,Lung_Opacity/masks/Lung_Opacity-5985.png,True,G08941,test,d3351dd194d1103f435d606e06db3910,8c3a130d4f79fce0
Lung_Opacity-5986.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5986.png,Lung_Opacity/masks/Lung_Opacity-5986.png,True,G08942,train,c8b860f3ce390bbbf4ecfffd29b546f2,d03f521f4b091d3d
Lung_Opacity-5987.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5987.png,Lung_Opacity/masks/Lung_Opacity-5987.png,True,G08943,val,842cd0093508271f6a9cb0203c25179c,d43f175d0c0f2f0a
Lung_Opacity-5988.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5988.png,Lung_Opacity/masks/Lung_Opacity-5988.png,True,G08944,val,fae63221af481d68524674e2745a3e75,c43c93419f4e2f59
Lung_Opacity-5989.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5989.png,Lung_Opacity/masks/Lung_Opacity-5989.png,True,G08945,val,2d0ad9c700e03562b699e9544e7719b4,d02b77c21e15c7e1
Lung_Opacity-599.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-599.png,Lung_Opacity/masks/Lung_Opacity-599.png,True,G08946,test,1dbe04d92ef407993d7994408b608fa1,cc2e134b0f5c6a6d
Lung_Opacity-5990.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5990.png,Lung_Opacity/masks/Lung_Opacity-5990.png,True,G08947,train,8382c81811ef117dcb5a6ede217397d3,d839434b4b4b5b4e
Lung_Opacity-5991.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5991.png,Lung_Opacity/masks/Lung_Opacity-5991.png,True,G08948,test,2ff834b35a2c84209f0b5d4ebc1fce32,c97f161e5e1b1a0a
Lung_Opacity-5992.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5992.png,Lung_Opacity/masks/Lung_Opacity-5992.png,True,G08949,train,59c4d852cfa397fbcd6e90ad7478b9d7,cc2ed34f0e4d1c6c
Lung_Opacity-5993.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5993.png,Lung_Opacity/masks/Lung_Opacity-5993.png,True,G08950,train,3ccc9160ba47f3233d7290765874d840,c17b875a3c5b8919
Lung_Opacity-5994.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5994.png,Lung_Opacity/masks/Lung_Opacity-5994.png,True,G08951,train,25fcc1a8fefcb103f18a54bfea46bf71,856e971b184d1aee
Lung_Opacity-5995.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5995.png,Lung_Opacity/masks/Lung_Opacity-5995.png,True,G08952,train,5fcdc79564a10aa5f8ceebebb3704223,dc35d38aaaaaa8aa
Lung_Opacity-5996.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5996.png,Lung_Opacity/masks/Lung_Opacity-5996.png,True,G08953,train,52fe86c169a438e590683bf27fcb98e8,d56e437a78313236
Lung_Opacity-5997.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5997.png,Lung_Opacity/masks/Lung_Opacity-5997.png,True,G08954,train,7ed7010fbac3c5c5ffed3f250ffdcd62,807d065e58796be9
Lung_Opacity-5998.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5998.png,Lung_Opacity/masks/Lung_Opacity-5998.png,True,G08955,train,ef85729c02ea8bb38263e8436e51b1ec,d43d420b0f4e6d6e
Lung_Opacity-5999.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-5999.png,Lung_Opacity/masks/Lung_Opacity-5999.png,True,G08956,test,463221548694f48bc9d80bfca29c72dc,8d2c075f0d6c6c6d
Lung_Opacity-6.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-6.png,Lung_Opacity/masks/Lung_Opacity-6.png,True,G08957,test,5fc358d227267cf09970fe2be9957901,d235161a1a3f1f1d
Lung_Opacity-60.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-60.png,Lung_Opacity/masks/Lung_Opacity-60.png,True,G08958,train,13f8952478f42ec7719c0ac1d6ffc547,997b45c617197a68
Lung_Opacity-600.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-600.png,Lung_Opacity/masks/Lung_Opacity-600.png,True,G08959,train,313de802183d45db147460d5a9f54825,913946871e5acf4d
Lung_Opacity-6000.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-6000.png,Lung_Opacity/masks/Lung_Opacity-6000.png,True,G08960,train,af3b3bdfcabbfab68bdf4308042e21f8,857f83d63a51451d
Lung_Opacity-6001.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-6001.png,Lung_Opacity/masks/Lung_Opacity-6001.png,True,G08961,val,1b8245ec63a2062347ef3113d3cd6d78,956a821b19197d5f
Lung_Opacity-6002.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-6002.png,Lung_Opacity/masks/Lung_Opacity-6002.png,True,G08962,train,7c9d03449cbd0b0f2a156b5c9e5ffb67,d42d4a5f0f5d0d0d
Lung_Opacity-6003.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-6003.png,Lung_Opacity/masks/Lung_Opacity-6003.png,True,G08963,train,5b6c2b351121f6659c30a09e79e688a6,913b121b3bbdb0b4
Lung_Opacity-6004.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-6004.png,Lung_Opacity/masks/Lung_Opacity-6004.png,True,G08964,train,92b4b7eec56c2a0381c72c863c7d1d5c,d42e61430f4e6f4d
Lung_Opacity-6005.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-6005.png,Lung_Opacity/masks/Lung_Opacity-6005.png,True,G08965,val,11fcee9149f54e3355bfd8859f131b27,d2ec2d9b29319333
Lung_Opacity-6006.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-6006.png,Lung_Opacity/masks/Lung_Opacity-6006.png,True,G08966,val,4c9ce0ad0f015f44a8afed6c30f2a5f7,85681e1a1b7b2d3d
Lung_Opacity-6007.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-6007.png,Lung_Opacity/masks/Lung_Opacity-6007.png,True,G08967,test,a101dcf7bb27e686a7b85150c4d7f591,897d861b5f097ba0
Lung_Opacity-6008.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-6008.png,Lung_Opacity/masks/Lung_Opacity-6008.png,True,G08968,train,b3aa42162675b900fd1afd1c62993b1d,d02fc34b49cb694d
Lung_Opacity-6009.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-6009.png,Lung_Opacity/masks/Lung_Opacity-6009.png,True,G08969,train,057058648a94e2738152b9d3d60de812,9139e6161a4d87fa
Lung_Opacity-601.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-601.png,Lung_Opacity/masks/Lung_Opacity-601.png,True,G08970,train,f04745b614b9b6bc7da163e3dd7d9e2e,d93995821f1d425f
Lung_Opacity-6010.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-6010.png,Lung_Opacity/masks/Lung_Opacity-6010.png,True,G08971,train,0c822069f89410f9e38bb9c0b61dffa7,c83c074a586de5db
Lung_Opacity-6011.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-6011.png,Lung_Opacity/masks/Lung_Opacity-6011.png,True,G08972,train,800ad56fa50b7d631cb8a313162eb246,d03143431e1fbbb9
Lung_Opacity-6012.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-6012.png,Lung_Opacity/masks/Lung_Opacity-6012.png,True,G08973,train,3c2a1cbb681cef159e086c6051b21437,986f061a4d7d485f
Lung_Opacity-602.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-602.png,Lung_Opacity/masks/Lung_Opacity-602.png,True,G08974,train,5476c10f491df3c632d5afa2caacd5fb,917846841a5bed6f
Lung_Opacity-603.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-603.png,Lung_Opacity/masks/Lung_Opacity-603.png,True,G08975,train,a59ecc2470f76bb498d43045955aacb1,dc6a535f4d0c2c0f
Lung_Opacity-604.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-604.png,Lung_Opacity/masks/Lung_Opacity-604.png,True,G08976,train,0c0524f12822fda0eb3727075c81ec8a,c92d861e1d4d5cd9
Lung_Opacity-605.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-605.png,Lung_Opacity/masks/Lung_Opacity-605.png,True,G08977,train,e9244c64349314a48db98cb277e48e5c,dc2e470f2839e1b3
Lung_Opacity-606.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-606.png,Lung_Opacity/masks/Lung_Opacity-606.png,True,G08978,train,738f8ad59ece3240c777f53b40b6f747,d43d4a074d1e2d6d
Lung_Opacity-607.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-607.png,Lung_Opacity/masks/Lung_Opacity-607.png,True,G08979,val,8f3989a550dbda5a2a163fcefc0a9801,c02bd2c14c6b7d79
Lung_Opacity-608.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-608.png,Lung_Opacity/masks/Lung_Opacity-608.png,True,G08980,train,22be4558c80612e980367a80050bb65e,82601f1f1f3f3a38
Lung_Opacity-609.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-609.png,Lung_Opacity/masks/Lung_Opacity-609.png,True,G08981,val,2556b0eaa65b09d29ba81b15c0fe1fba,9568075f2a7d7868
Lung_Opacity-61.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-61.png,Lung_Opacity/masks/Lung_Opacity-61.png,True,G08982,train,71667d4e410220021c95240bc5373685,cc7412c34b37393d
Lung_Opacity-610.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-610.png,Lung_Opacity/masks/Lung_Opacity-610.png,True,G08983,test,07dbea70957f2a02ddd07d53ffb03f4c,8179d71615c65e71
Lung_Opacity-611.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-611.png,Lung_Opacity/masks/Lung_Opacity-611.png,True,G08984,test,2a4ba882bd01b72f7427930440cf821a,dc2dd343034f5e0d
Lung_Opacity-612.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-612.png,Lung_Opacity/masks/Lung_Opacity-612.png,True,G08985,train,ba700019357738fe736c618bce13322f,943c43409fda999f
Lung_Opacity-613.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-613.png,Lung_Opacity/masks/Lung_Opacity-613.png,True,G08986,val,02c8632c5d995f837ccdf0366fd975e1,dc4b470d8c0f8797
Lung_Opacity-614.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-614.png,Lung_Opacity/masks/Lung_Opacity-614.png,True,G08987,test,a6392d122a9869c4e93f523e2f7adeee,940e471f59c97ce4
Lung_Opacity-615.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-615.png,Lung_Opacity/masks/Lung_Opacity-615.png,True,G08988,train,6e4932dbe4f522ceaf7aefc469bf36c5,c52e434b4c6c72fc
Lung_Opacity-616.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-616.png,Lung_Opacity/masks/Lung_Opacity-616.png,True,G08989,train,100ca4838877b559b7e414f362918fa5,8c2e030f0d5d5e5f
Lung_Opacity-617.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-617.png,Lung_Opacity/masks/Lung_Opacity-617.png,True,G08990,train,d881281eeaf150b24bfd4ed2d7ffeff5,d52ad25f1c1a692e
Lung_Opacity-618.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-618.png,Lung_Opacity/masks/Lung_Opacity-618.png,True,G08991,train,639d34ac6dd52f58ed3129617e1e2dbd,996d061ac51ad9cf
Lung_Opacity-619.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-619.png,Lung_Opacity/masks/Lung_Opacity-619.png,True,G08992,train,021a41049d398132bc5cac8c885adac5,9879c74e1e494f0d
Lung_Opacity-62.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-62.png,Lung_Opacity/masks/Lung_Opacity-62.png,True,G08993,train,8371c878b7e8f20b8f2f712b6ad2c733,cc7dd70b0a4f380e
Lung_Opacity-620.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-620.png,Lung_Opacity/masks/Lung_Opacity-620.png,True,G08994,train,96e01ff8752fe0efd8ae4e8080939b50,c53d1683111f4f1f
Lung_Opacity-621.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-621.png,Lung_Opacity/masks/Lung_Opacity-621.png,True,G08995,train,f2d78ca735e3f7d0fd806ee0591c5554,814106de1f5f594f
Lung_Opacity-622.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-622.png,Lung_Opacity/masks/Lung_Opacity-622.png,True,G08996,train,c7af6165fbb81c9ba18f38af7b0f0d80,d42f435f0d474e1c
Lung_Opacity-623.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-623.png,Lung_Opacity/masks/Lung_Opacity-623.png,True,G08997,train,ab77bd5dbcc419631a277d3e22a28da0,917e04184d7f47ce
Lung_Opacity-624.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-624.png,Lung_Opacity/masks/Lung_Opacity-624.png,True,G08998,train,34f90747677ed4fec47c964c6cba5f6f,856ade80125b7b7a
Lung_Opacity-625.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-625.png,Lung_Opacity/masks/Lung_Opacity-625.png,True,G08999,train,13f030e9998785d49c6ba39d7baf7976,cc39c2c31b4f630f
Lung_Opacity-626.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-626.png,Lung_Opacity/masks/Lung_Opacity-626.png,True,G09000,train,fbbd200aa9c3ae27c4a93341d444024d,d42d538b4e2c3a3d
Lung_Opacity-627.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-627.png,Lung_Opacity/masks/Lung_Opacity-627.png,True,G09001,train,6bc63fbb5ef9a42cd189586a3b960d31,817fc7961a473e18
Lung_Opacity-628.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-628.png,Lung_Opacity/masks/Lung_Opacity-628.png,True,G09002,val,aa8c07e9c246eaf9b49f1283f476a576,817c0c58595dc7e7
Lung_Opacity-629.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-629.png,Lung_Opacity/masks/Lung_Opacity-629.png,True,G09003,test,59010768f5b1a97c335f31305f20b84e,881d061e1a5f7c7b
Lung_Opacity-63.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-63.png,Lung_Opacity/masks/Lung_Opacity-63.png,True,G09004,test,7eb68cd375786fc603a97bfb2e8b999e,d43b431f1e0a1d3e
Lung_Opacity-630.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-630.png,Lung_Opacity/masks/Lung_Opacity-630.png,True,G09005,test,f6da17b034fecb0269a2543f8ae63c2c,dc2ec0060f6d3d3e
Lung_Opacity-631.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-631.png,Lung_Opacity/masks/Lung_Opacity-631.png,True,G09006,train,91ea7ef88f92244012f999eeaa866920,c03f430a1e4b5f1f
Lung_Opacity-632.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-632.png,Lung_Opacity/masks/Lung_Opacity-632.png,True,G09007,val,5d3cafb22aee3f0f411a2bee175c5cf1,95788e469859e97c
Lung_Opacity-633.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-633.png,Lung_Opacity/masks/Lung_Opacity-633.png,True,G09008,train,dac0f24933e3bbd657ab682223c2e4eb,d52f024b1d6c6abc
Lung_Opacity-634.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-634.png,Lung_Opacity/masks/Lung_Opacity-634.png,True,G09009,val,24cd3197bad065bc557c09eb62b9013b,d42df3434a4e6c6c
Lung_Opacity-635.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-635.png,Lung_Opacity/masks/Lung_Opacity-635.png,True,G09010,train,c0236e28eaa3863a831fd4f987b39044,9d5dc6061f4d131b
Lung_Opacity-636.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-636.png,Lung_Opacity/masks/Lung_Opacity-636.png,True,G09011,train,e7624e8fe78c143559e0a4a3e9642516,d43e47430b0a4f3f
Lung_Opacity-637.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-637.png,Lung_Opacity/masks/Lung_Opacity-637.png,True,G09012,train,61277bcecf6358ce4f6cb5d6c0af7ac1,c27d7fe2b5441a09
Lung_Opacity-638.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-638.png,Lung_Opacity/masks/Lung_Opacity-638.png,True,G09013,train,96ca0b4fb89c75073a2f6c12d65a8cee,d81a961e1cafaf88
Lung_Opacity-639.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-639.png,Lung_Opacity/masks/Lung_Opacity-639.png,True,G09014,train,ff5227f0b0eb2a7a5eca94bc2df651e7,99691e1f1d5e69a0
Lung_Opacity-64.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-64.png,Lung_Opacity/masks/Lung_Opacity-64.png,True,G09015,train,b71e1fa7fc48c1f8d099695143eb87c0,d529070a1d5d3d5d
Lung_Opacity-640.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-640.png,Lung_Opacity/masks/Lung_Opacity-640.png,True,G09016,val,ed3b9f1dfa31ed55c7fd569b3fd13bc6,d42ea10b8b2b2f2f
Lung_Opacity-641.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-641.png,Lung_Opacity/masks/Lung_Opacity-641.png,True,G09017,train,f2cfdf6e6aefd1b66c838c3bd99560fc,c5297a5350475b5d
Lung_Opacity-642.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-642.png,Lung_Opacity/masks/Lung_Opacity-642.png,True,G09018,train,7936c4cc7835b37f30765797cb7f4dba,dc79531b1e3ab034
Lung_Opacity-643.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-643.png,Lung_Opacity/masks/Lung_Opacity-643.png,True,G09019,train,62389f30be1537300fd45cc50b1d7f7f,dc2ed7430d470f4c
Lung_Opacity-644.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-644.png,Lung_Opacity/masks/Lung_Opacity-644.png,True,G09020,train,65b84d03bef2d49d1d3fe07029375a0c,817a961a595b5b1b
Lung_Opacity-645.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-645.png,Lung_Opacity/masks/Lung_Opacity-645.png,True,G09021,train,9d967b985895a5d854a7355823682aa6,883f05065e5f5a79
Lung_Opacity-646.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-646.png,Lung_Opacity/masks/Lung_Opacity-646.png,True,G09022,val,fd58b7c3f807297c092ecb9e49304c83,8178161e5b193b3f
Lung_Opacity-647.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-647.png,Lung_Opacity/masks/Lung_Opacity-647.png,True,G09023,train,c5a3fa1d4690d4086f7913c8d9f2cdd6,813f96021d175f57
Lung_Opacity-648.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-648.png,Lung_Opacity/masks/Lung_Opacity-648.png,True,G09024,test,723c12e1abcbe9e2dbd70cd85da0a378,983f5a1a49783ce3
Lung_Opacity-649.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-649.png,Lung_Opacity/masks/Lung_Opacity-649.png,True,G09025,test,9e52586c501ffb2268f6104ce9f8d6e7,d02f282a2b376737
Lung_Opacity-65.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-65.png,Lung_Opacity/masks/Lung_Opacity-65.png,True,G09026,val,30f55233ad0a41bd9459573f61b3388b,8d2c465c4c4aedeb
Lung_Opacity-650.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-650.png,Lung_Opacity/masks/Lung_Opacity-650.png,True,G09027,test,4ebe850977806e0b1574f653aefcfa2c,9178071e5f4b78e8
Lung_Opacity-651.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-651.png,Lung_Opacity/masks/Lung_Opacity-651.png,True,G09028,train,74d5dee6ba37630fc3b41cc2c47eae4c,901a1c1a5f5a7afa
Lung_Opacity-652.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-652.png,Lung_Opacity/masks/Lung_Opacity-652.png,True,G09029,train,ee25f82d0f58f4ee8f3a5eb71b65fe83,cc2a5b5f2c0d0e8f
Lung_Opacity-653.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-653.png,Lung_Opacity/masks/Lung_Opacity-653.png,True,G09030,train,1879d8388941edfe3a5c5c62af6f1294,d03d53434f4e4e4e
Lung_Opacity-654.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-654.png,Lung_Opacity/masks/Lung_Opacity-654.png,True,G09031,train,cbf113549516670435fafe5f7f55848d,913947961a6cf939
Lung_Opacity-655.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-655.png,Lung_Opacity/masks/Lung_Opacity-655.png,True,G09032,train,be1b4b27069c1fbbf2d0da4a80d45453,e92dd61bc7421f48
Lung_Opacity-656.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-656.png,Lung_Opacity/masks/Lung_Opacity-656.png,True,G09033,train,6d4ba442a9797ed5d99697069e4204f5,dd2cc2535f490f1c
Lung_Opacity-657.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-657.png,Lung_Opacity/masks/Lung_Opacity-657.png,True,G09034,train,3abb124edb3b8d5069d4ee35a1f5167c,de3d430b0f2e0d0f
Lung_Opacity-658.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-658.png,Lung_Opacity/masks/Lung_Opacity-658.png,True,G09035,train,60dfcf329df1cb89f070e097396c3ea3,917bc75759095a1a
Lung_Opacity-659.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-659.png,Lung_Opacity/masks/Lung_Opacity-659.png,True,G09036,train,bd142621d3d5803a9d54e82258ceeea1,9179461b191f3ba9
Lung_Opacity-66.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-66.png,Lung_Opacity/masks/Lung_Opacity-66.png,True,G09037,train,6e530d789d785e2dd51f2dd10afa9931,d939c71a4d2d1a69
Lung_Opacity-660.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-660.png,Lung_Opacity/masks/Lung_Opacity-660.png,True,G09038,train,a6133c98380236d8c8c6626597ba815c,d47fcb0a1b4b4b0c
Lung_Opacity-661.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-661.png,Lung_Opacity/masks/Lung_Opacity-661.png,True,G09039,train,2f56a0682297f442d3e8aa4c68c6e65f,d03f5200113fb7b7
Lung_Opacity-662.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-662.png,Lung_Opacity/masks/Lung_Opacity-662.png,True,G09040,train,97abdae3a15d14a22405d6ce59865a9b,885a121a1c5f5f3f
Lung_Opacity-663.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-663.png,Lung_Opacity/masks/Lung_Opacity-663.png,True,G09041,val,d4ad93abf0c75f00abc3f31e927d1c03,95780e0e5f2969f8
Lung_Opacity-664.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-664.png,Lung_Opacity/masks/Lung_Opacity-664.png,True,G09042,train,fc36822c36d56ef330469c3d0d91c3da,df3a141e181f781e
Lung_Opacity-665.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-665.png,Lung_Opacity/masks/Lung_Opacity-665.png,True,G09043,val,9c203f6c677536970bdd8a6c5bca48b0,9978c71e5acb38a8
Lung_Opacity-666.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-666.png,Lung_Opacity/masks/Lung_Opacity-666.png,True,G09044,val,429ab8307bd04ee7717994111dde8536,dd38465f093e4a1d
Lung_Opacity-667.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-667.png,Lung_Opacity/masks/Lung_Opacity-667.png,True,G09045,test,f52721e2d6f9425f5c95b22b0c9478d4,d03d110b3b3f391b
Lung_Opacity-668.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-668.png,Lung_Opacity/masks/Lung_Opacity-668.png,True,G09046,test,6bd115e7933debdd058228ae2447cf07,8d2f824b055f4b57
Lung_Opacity-669.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-669.png,Lung_Opacity/masks/Lung_Opacity-669.png,True,G09047,test,b8f56f9ae61941b4dd6a48ac8da79bae,813f161e1e5c595d
Lung_Opacity-67.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-67.png,Lung_Opacity/masks/Lung_Opacity-67.png,True,G09048,train,1cec58e6718873a139c75bc42f0dd18a,997bc61715471e58
Lung_Opacity-670.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-670.png,Lung_Opacity/masks/Lung_Opacity-670.png,True,G09049,train,3528651508c717027685e979ccf3b47f,d83d4b921f4b8e1c
Lung_Opacity-671.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-671.png,Lung_Opacity/masks/Lung_Opacity-671.png,True,G09050,train,fa46c0158975942616b3b748c46a6af8,d03fc7d61809595d
Lung_Opacity-672.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-672.png,Lung_Opacity/masks/Lung_Opacity-672.png,True,G09051,train,0a4363fde0d96ea62486d1645acf4aa1,843e07424f5f4f6c
Lung_Opacity-673.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-673.png,Lung_Opacity/masks/Lung_Opacity-673.png,True,G09052,train,5f13fdfa71b069edf333d73b723747eb,9571489e5e9a7a98
Lung_Opacity-674.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-674.png,Lung_Opacity/masks/Lung_Opacity-674.png,True,G09053,train,5e6c0d682d6358876407d85e4b8a66e7,947aa35259c5459f
Lung_Opacity-675.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-675.png,Lung_Opacity/masks/Lung_Opacity-675.png,True,G09054,train,015df503b0020214ad186e9d5824bf82,c17e164d5e4c4e78
Lung_Opacity-676.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-676.png,Lung_Opacity/masks/Lung_Opacity-676.png,True,G09055,train,a4d1fffe690720b44d75829e19b804dc,d47e83051d1f1b5c
Lung_Opacity-677.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-677.png,Lung_Opacity/masks/Lung_Opacity-677.png,True,G09056,train,cd0ce597a17b1b086ff080c0ba2713ad,82715a5e1f585a5b
Lung_Opacity-678.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-678.png,Lung_Opacity/masks/Lung_Opacity-678.png,True,G09057,train,75301e12cf980776c570aaf97b1e7624,93788410db7a7ace
Lung_Opacity-679.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-679.png,Lung_Opacity/masks/Lung_Opacity-679.png,True,G09058,train,bbbb57817ded89a9f5d981856fa25d41,913b1a1e1e1d595d
Lung_Opacity-68.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-68.png,Lung_Opacity/masks/Lung_Opacity-68.png,True,G09059,train,30120f42634a7e22befc683a14afe7f3,d07b52025a5a5bbb
Lung_Opacity-680.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-680.png,Lung_Opacity/masks/Lung_Opacity-680.png,True,G09060,train,47c4004fe1ce16189769f25475cf3378,d52ec31f1d4c4d4c
Lung_Opacity-681.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-681.png,Lung_Opacity/masks/Lung_Opacity-681.png,True,G09061,train,b508d2840dc74d2f5102f40ae5a32911,c03f5a153a3f6e28
Lung_Opacity-682.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-682.png,Lung_Opacity/masks/Lung_Opacity-682.png,True,G09062,val,5fca87f7779fae1f309b907501ceec88,c97b47161e0d4e1b
Lung_Opacity-683.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-683.png,Lung_Opacity/masks/Lung_Opacity-683.png,True,G09063,test,3a8e3b22804b2b99068267e1089e2327,887d80df9dc59516
Lung_Opacity-684.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-684.png,Lung_Opacity/masks/Lung_Opacity-684.png,True,G09064,val,3088991d252ec91caad8e4755266e607,807b565a59191f1f
Lung_Opacity-685.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-685.png,Lung_Opacity/masks/Lung_Opacity-685.png,True,G09065,train,88023487da81e02edea1569ee2380007,813b54165e5a7979
Lung_Opacity-686.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-686.png,Lung_Opacity/masks/Lung_Opacity-686.png,True,G09066,val,22f84ee59ca38c4d5de6fd7c8781d5a5,c47d0b0e7c287df0
Lung_Opacity-687.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-687.png,Lung_Opacity/masks/Lung_Opacity-687.png,True,G09067,test,aecc36c0c4f002f064800563b94d3965,cc3ed349574b4c4a
Lung_Opacity-688.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-688.png,Lung_Opacity/masks/Lung_Opacity-688.png,True,G09068,train,cb039039acb2ab2f46a01671ac27312f,807ea713835f1e1e
Lung_Opacity-689.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-689.png,Lung_Opacity/masks/Lung_Opacity-689.png,True,G09069,train,dafd0f6e710323c6e2826fcb31081da0,9138177f7a6a6262
Lung_Opacity-69.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-69.png,Lung_Opacity/masks/Lung_Opacity-69.png,True,G09070,train,85c5ba6c831c5d98cbcfa2e7ed248ea6,d02f5f4f2c3c10ad
Lung_Opacity-690.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-690.png,Lung_Opacity/masks/Lung_Opacity-690.png,True,G09071,test,9c86233f11aa1046507b12d4c4bc9ea4,d03b5d585a1b1b1b
Lung_Opacity-691.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-691.png,Lung_Opacity/masks/Lung_Opacity-691.png,True,G09072,train,4707a25797d9de2adb60e7d79b629dec,dd2e030f0c4e4b7b
Lung_Opacity-692.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-692.png,Lung_Opacity/masks/Lung_Opacity-692.png,True,G09073,train,d245d890bf5dded61fb181c38c59d84d,95680f4a1b1b0f3f
Lung_Opacity-693.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-693.png,Lung_Opacity/masks/Lung_Opacity-693.png,True,G09074,val,37beb1e43ae66040b68fd85f6265082c,80281f1b1f5b5b5b
Lung_Opacity-694.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-694.png,Lung_Opacity/masks/Lung_Opacity-694.png,True,G09075,train,6fce988b829e3401c6f701000eec563e,907f4e5d49495cc9
Lung_Opacity-695.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-695.png,Lung_Opacity/masks/Lung_Opacity-695.png,True,G09076,train,d9a2d5777855c42c8dae9790b57e4300,917f54101b1f3537
Lung_Opacity-696.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-696.png,Lung_Opacity/masks/Lung_Opacity-696.png,True,G09077,train,c85b87f3abe6ce482f3270d31e94be18,c12dbe92c2cbecb0
Lung_Opacity-697.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-697.png,Lung_Opacity/masks/Lung_Opacity-697.png,True,G09078,train,8f7c407faa5f2521270b648638cd8ee9,803d501f572b3f2d
Lung_Opacity-698.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-698.png,Lung_Opacity/masks/Lung_Opacity-698.png,True,G09079,val,3248c2400d581fbe86b087fac1b61cc6,917e465a1968796d
Lung_Opacity-699.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-699.png,Lung_Opacity/masks/Lung_Opacity-699.png,True,G09080,train,bdb58f1a96b7f6d7ad780e655679b96b,9579461a09795d5d
Lung_Opacity-7.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-7.png,Lung_Opacity/masks/Lung_Opacity-7.png,True,G09081,train,64f4aa1f38afdb4ca6349a1ee5d13358,d52e170a0f2c2e6f
Lung_Opacity-70.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-70.png,Lung_Opacity/masks/Lung_Opacity-70.png,True,G09082,train,98c72f4ebbb102411b071d38809bca2f,81291e975a7bf868
Lung_Opacity-700.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-700.png,Lung_Opacity/masks/Lung_Opacity-700.png,True,G09083,train,e0d6f30bd9afc7589466894625e5e711,c426034fe2e5e2f3
Lung_Opacity-701.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-701.png,Lung_Opacity/masks/Lung_Opacity-701.png,True,G09084,train,fdb6223c4f25c1aafc02912d355e636e,c83d0b2d247476f5
Lung_Opacity-702.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-702.png,Lung_Opacity/masks/Lung_Opacity-702.png,True,G09085,val,943014d510149d15d5147c105fe01b03,8139151f5e1e5c5e
Lung_Opacity-703.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-703.png,Lung_Opacity/masks/Lung_Opacity-703.png,True,G09086,test,34ae47034a1b0f1339f985218d0bff3a,817a13175e5a1b3b
Lung_Opacity-704.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-704.png,Lung_Opacity/masks/Lung_Opacity-704.png,True,G09087,test,2edf26a16904be75aed932bba92e64cf,817b66969096ce5e
Lung_Opacity-705.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-705.png,Lung_Opacity/masks/Lung_Opacity-705.png,True,G09088,val,bd0e80b076f8436f09e3d3df3c2e9f09,d43c2a0e2e2e6f4b
Lung_Opacity-706.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-706.png,Lung_Opacity/masks/Lung_Opacity-706.png,True,G09089,test,b6b48115536b49ec2d72d23811a3909a,c62d484e4f8f0da7
Lung_Opacity-707.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-707.png,Lung_Opacity/masks/Lung_Opacity-707.png,True,G09090,train,1efaddb9d243d6dc5f75c2de51eee485,8d2c161e4d5f7869
Lung_Opacity-708.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-708.png,Lung_Opacity/masks/Lung_Opacity-708.png,True,G09091,train,cdb9f44f077a5c78a736bf8c227841d1,884e065dfdf6a2a2
Lung_Opacity-709.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-709.png,Lung_Opacity/masks/Lung_Opacity-709.png,True,G09092,train,283d215aa361276e2c664e11123039da,8528175e1e5f5a5a
Lung_Opacity-71.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-71.png,Lung_Opacity/masks/Lung_Opacity-71.png,True,G09093,train,e91f61d14171ece580a0cebf37d5c683,dc2c528b0cade8bd
Lung_Opacity-710.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-710.png,Lung_Opacity/masks/Lung_Opacity-710.png,True,G09094,train,3aba26e45670b95ed555d60afe921fa1,807e070e5d7d4d4d
Lung_Opacity-711.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-711.png,Lung_Opacity/masks/Lung_Opacity-711.png,True,G09095,train,cc04244e388a118cc906f70454079938,856c0e586de5b1b3
Lung_Opacity-712.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-712.png,Lung_Opacity/masks/Lung_Opacity-712.png,True,G09096,train,2d163e5135d1489569909a63249ba435,90394f4e1e4f0f1b
Lung_Opacity-713.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-713.png,Lung_Opacity/masks/Lung_Opacity-713.png,True,G09097,train,b076d93feb86821f94dc65b3238e6af9,837f8410c75f551d
Lung_Opacity-714.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-714.png,Lung_Opacity/masks/Lung_Opacity-714.png,True,G09098,train,e6da787f74b4b75c3cdf0d1319f154ff,887f86161b555d1d
Lung_Opacity-715.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-715.png,Lung_Opacity/masks/Lung_Opacity-715.png,True,G09099,train,807176874f3f12fbd9d21bb195a220d4,817a84d49ae9f7d0
Lung_Opacity-716.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-716.png,Lung_Opacity/masks/Lung_Opacity-716.png,True,G09100,train,20f0422352cd01bde793c622a23f93d6,857a0c1a1b5d5d79
Lung_Opacity-717.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-717.png,Lung_Opacity/masks/Lung_Opacity-717.png,True,G09101,train,f051425114d39889255b4060b73062a7,c43dd34b2e2a4b0f
Lung_Opacity-718.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-718.png,Lung_Opacity/masks/Lung_Opacity-718.png,True,G09102,train,b02a4d58f15b34c4bc2d1a3da39339d3,d427414d466eeaea
Lung_Opacity-719.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-719.png,Lung_Opacity/masks/Lung_Opacity-719.png,True,G09103,val,8afbf40474f3bd28a6b6823d89586b25,d42f430b2e2f2d4e
Lung_Opacity-72.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-72.png,Lung_Opacity/masks/Lung_Opacity-72.png,True,G09104,test,704eabb29adb262e26a8cb01dafcf564,dd2cd20b1f4a0f3c
Lung_Opacity-720.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-720.png,Lung_Opacity/masks/Lung_Opacity-720.png,True,G09105,val,3a5a65490e1b64429066da0d13ba1f2f,d42f055d1d045f1f
Lung_Opacity-721.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-721.png,Lung_Opacity/masks/Lung_Opacity-721.png,True,G09106,test,937997bf7d65b0cdb16fcfaabab8f611,d17b41c796924357
Lung_Opacity-722.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-722.png,Lung_Opacity/masks/Lung_Opacity-722.png,True,G09107,train,c46b108e1d1d1fb877c0d9b7eee695c1,85700e1f1f5d1c1f
Lung_Opacity-723.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-723.png,Lung_Opacity/masks/Lung_Opacity-723.png,True,G09108,test,0d744c4ffccde1c9b1e07c7f6f1c0cc0,cd2a570a5f1a4a2f
Lung_Opacity-724.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-724.png,Lung_Opacity/masks/Lung_Opacity-724.png,True,G09109,train,d0e35bc19308ac1360d061c36ca2dddc,c43d560b4f0e6e2e
Lung_Opacity-725.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-725.png,Lung_Opacity/masks/Lung_Opacity-725.png,True,G09110,val,b6c08e85db213e7de415c528c7e1a5a5,d43c434d2f0f1e1e
Lung_Opacity-726.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-726.png,Lung_Opacity/masks/Lung_Opacity-726.png,True,G09111,train,a6043bdb9a852e2806e3962e653ce384,c02b5551b95b5f0b
Lung_Opacity-727.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-727.png,Lung_Opacity/masks/Lung_Opacity-727.png,True,G09112,train,a6acb079dad720bd23b62351a1b348cf,c43d5212cb0b5f1f
Lung_Opacity-728.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-728.png,Lung_Opacity/masks/Lung_Opacity-728.png,True,G09113,train,a51a21ea9c542c759f90863d93af0a8f,ff5a1e1a1a1bd850
Lung_Opacity-729.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-729.png,Lung_Opacity/masks/Lung_Opacity-729.png,True,G09114,train,5d7bea7d54708952ab5f034d726f34da,c03d0a0d0f0f3f3f
Lung_Opacity-73.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-73.png,Lung_Opacity/masks/Lung_Opacity-73.png,True,G09115,train,24eb19715b617225d445808732241dd8,d07e2f57f4341246
Lung_Opacity-730.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-730.png,Lung_Opacity/masks/Lung_Opacity-730.png,True,G09116,train,739c9c0f1c115d07dfad62e8770d1bbb,dc6f474f4c4c462a
Lung_Opacity-731.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-731.png,Lung_Opacity/masks/Lung_Opacity-731.png,True,G09117,train,79f0397ff69856339c23327fc1565a1b,8378de1b790ff801
Lung_Opacity-732.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-732.png,Lung_Opacity/masks/Lung_Opacity-732.png,True,G09118,train,14c79837b8db8005d9e6018902bb6752,c62b494a272f2d3d
Lung_Opacity-733.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-733.png,Lung_Opacity/masks/Lung_Opacity-733.png,True,G09119,train,31351e541d57082694276c92d16ce1bd,8938171e5e5a787a
Lung_Opacity-734.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-734.png,Lung_Opacity/masks/Lung_Opacity-734.png,True,G09120,train,59d4d4abd6f24bbd16a0da4ba8277690,807b04151f5f1b5f
Lung_Opacity-735.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-735.png,Lung_Opacity/masks/Lung_Opacity-735.png,True,G09121,train,e860a8106e35d7cc169ed02ad8702c8a,d17e0717165a1e5a
Lung_Opacity-736.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-736.png,Lung_Opacity/masks/Lung_Opacity-736.png,True,G09122,train,7d29cc808ed5f5d92f0c5023779f3221,81384c3f7f767062
Lung_Opacity-737.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-737.png,Lung_Opacity/masks/Lung_Opacity-737.png,True,G09123,val,f14b979fba65c29359fc7e9b6b04c6e2,813b161a1a597b5f
Lung_Opacity-738.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-738.png,Lung_Opacity/masks/Lung_Opacity-738.png,True,G09124,val,d626797b6bce7d111b082926eb17bcd0,d52e125b194f0b4f
Lung_Opacity-739.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-739.png,Lung_Opacity/masks/Lung_Opacity-739.png,True,G09125,test,07de50d990311090eae91f79ed6df8bb,d42e50c1434f5fcb
Lung_Opacity-74.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-74.png,Lung_Opacity/masks/Lung_Opacity-74.png,True,G09126,train,dab9a26b36d15eceb214340217b43304,913b68405d7b7978
Lung_Opacity-740.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-740.png,Lung_Opacity/masks/Lung_Opacity-740.png,True,G09127,test,f17d824be7b2133f179b994050b54ab7,d46ea3d31f0e0e0e
Lung_Opacity-741.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-741.png,Lung_Opacity/masks/Lung_Opacity-741.png,True,G09128,test,771f6e884a263d4ed13f0d480d8ef3fa,896892591f5b1d79
Lung_Opacity-742.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-742.png,Lung_Opacity/masks/Lung_Opacity-742.png,True,G09129,train,bbd93efbdf52c9fe805ae70892ef30d6,9968061e1b697d7a
Lung_Opacity-743.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-743.png,Lung_Opacity/masks/Lung_Opacity-743.png,True,G09130,train,8ced7dc932e4ff2433ff2f14d2b337fe,d179c21a5b1a3a5b
Lung_Opacity-744.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-744.png,Lung_Opacity/masks/Lung_Opacity-744.png,True,G09131,train,9bcba1f5d66db0b2d48d381f2b925b33,c03f7d040d4d1f1f
Lung_Opacity-745.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-745.png,Lung_Opacity/masks/Lung_Opacity-745.png,True,G09132,val,0a6d6683161005a3bb312d6c9055963e,813b56045f1b3b3b
Lung_Opacity-746.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-746.png,Lung_Opacity/masks/Lung_Opacity-746.png,True,G09133,train,fd9f5a0eff8246a0d66225e1d0027dcf,dc2f0b0d4d1e8a1f
Lung_Opacity-747.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-747.png,Lung_Opacity/masks/Lung_Opacity-747.png,True,G09134,train,8be471feb73ae92acc730471e0a4a081,c81b135b183a5d3f
Lung_Opacity-748.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-748.png,Lung_Opacity/masks/Lung_Opacity-748.png,True,G09135,train,5fb90307472692f33ae3cb9fecd315c8,857f150e0a5d7a6a
Lung_Opacity-749.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-749.png,Lung_Opacity/masks/Lung_Opacity-749.png,True,G09136,train,3b41f27fdad54d6bada2059898c77b99,9968161a595d595f
Lung_Opacity-75.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-75.png,Lung_Opacity/masks/Lung_Opacity-75.png,True,G09137,train,a3d56d580df3e8becce237d78e8a5bab,c43d1243164f1f6f
Lung_Opacity-750.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-750.png,Lung_Opacity/masks/Lung_Opacity-750.png,True,G09138,train,0360289f16631be67b135d461ef63159,d52fda4c0e0f0e0f
Lung_Opacity-751.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-751.png,Lung_Opacity/masks/Lung_Opacity-751.png,True,G09139,train,d1e2866bb5fcb4b4e3d0f43f8b154b07,c03d5b4b0f3a1e1b
Lung_Opacity-752.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-752.png,Lung_Opacity/masks/Lung_Opacity-752.png,True,G09140,train,acfb2dbfa134f8b74cb2b948cf03f43c,d46f03cb6f756060
Lung_Opacity-753.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-753.png,Lung_Opacity/masks/Lung_Opacity-753.png,True,G09141,val,10111e39086a6ada05cd73f341a9d193,d839531a1a383dbd
Lung_Opacity-754.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-754.png,Lung_Opacity/masks/Lung_Opacity-754.png,True,G09142,train,49c608e519b04d654d260b2ff3844a1e,817a165f69e5a2a6
Lung_Opacity-755.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-755.png,Lung_Opacity/masks/Lung_Opacity-755.png,True,G09143,train,207831dd39da6c492de9b7f8c60cf86e,c73776780f0b5868
Lung_Opacity-756.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-756.png,Lung_Opacity/masks/Lung_Opacity-756.png,True,G09144,val,1476a435ef64eec2a7c246b4dd48303c,c62d93090f0f2f6e
Lung_Opacity-757.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-757.png,Lung_Opacity/masks/Lung_Opacity-757.png,True,G09145,train,a02e0b9214bc5dcb2e521a663af9274a,d02d4f0e2daf8786
Lung_Opacity-758.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-758.png,Lung_Opacity/masks/Lung_Opacity-758.png,True,G09146,train,08721552c37cb1724fbd86cc1cd5b128,c16b52595a0b5b1f
Lung_Opacity-759.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-759.png,Lung_Opacity/masks/Lung_Opacity-759.png,True,G09147,test,b0ab3faf86b1eb330fa33178f4ace0fa,d43fa555075e170a
Lung_Opacity-76.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-76.png,Lung_Opacity/masks/Lung_Opacity-76.png,True,G09148,test,6ace1f15f4f325d5af6572cfb57e7643,cdae13c12cd46a97
Lung_Opacity-760.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-760.png,Lung_Opacity/masks/Lung_Opacity-760.png,True,G09149,train,8517582c1b5e76e72d2dc6b074a3f82e,c43d5a4b086f4f2e
Lung_Opacity-761.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-761.png,Lung_Opacity/masks/Lung_Opacity-761.png,True,G09150,test,991bfbb169426ea48c94391148348e16,836b0d5cd6d851d3
Lung_Opacity-762.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-762.png,Lung_Opacity/masks/Lung_Opacity-762.png,True,G09151,train,7234d6ddfee5f440f3f8a532fc109563,8d3d06c70b1f4d4b
Lung_Opacity-763.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-763.png,Lung_Opacity/masks/Lung_Opacity-763.png,True,G09152,train,02f4ce1dc8052a7c764a5171e2bafed5,c13f56465e1c5e1c
Lung_Opacity-764.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-764.png,Lung_Opacity/masks/Lung_Opacity-764.png,True,G09153,train,21ae86bcee371db16ad7f127eb6938a2,8a2fe56d6e0a6e0c
Lung_Opacity-765.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-765.png,Lung_Opacity/masks/Lung_Opacity-765.png,True,G09154,train,01cc1bdb239b7554c674c3502743e261,983a45474f47175e
Lung_Opacity-766.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-766.png,Lung_Opacity/masks/Lung_Opacity-766.png,True,G09155,train,86442db1a74bef6568cac17e5864d452,d02f5e180f2d2cbd
Lung_Opacity-767.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-767.png,Lung_Opacity/masks/Lung_Opacity-767.png,True,G09156,train,e5441bff5d59024a0fd944ab3bc57dfe,80a85f4578535d5f
Lung_Opacity-768.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-768.png,Lung_Opacity/masks/Lung_Opacity-768.png,True,G09157,train,c917dcc76f1ae91971312c8289aee5f6,902d475f1d495d5c
Lung_Opacity-769.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-769.png,Lung_Opacity/masks/Lung_Opacity-769.png,True,G09158,train,ad05fdae80c6fa1c3db8f0864bca525f,907f9610195f1f17
Lung_Opacity-77.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-77.png,Lung_Opacity/masks/Lung_Opacity-77.png,True,G09159,train,e3dd0d6ca5d305e4dcc0d2af474d7998,99784507165a5f1f
Lung_Opacity-770.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-770.png,Lung_Opacity/masks/Lung_Opacity-770.png,True,G09160,train,6b2c3f37f253cfa183506ae26d7b6e8a,956f075d49686a6a
Lung_Opacity-771.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-771.png,Lung_Opacity/masks/Lung_Opacity-771.png,True,G09161,train,326c95103447b4910cf3de479f1a4dc0,9169c65e5a491b79
Lung_Opacity-772.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-772.png,Lung_Opacity/masks/Lung_Opacity-772.png,True,G09162,train,41bee777a611b2bcd0f82f852982c681,913f0f19596a68ea
Lung_Opacity-773.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-773.png,Lung_Opacity/masks/Lung_Opacity-773.png,True,G09163,val,73f8873b6e9b1633121a025b8bc2afa9,917a161b7b38eaa4
Lung_Opacity-774.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-774.png,Lung_Opacity/masks/Lung_Opacity-774.png,True,G09164,val,94fc82fe0af3f220cf89d07800bf7ad5,dc28df4a1d1ce8e4
Lung_Opacity-775.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-775.png,Lung_Opacity/masks/Lung_Opacity-775.png,True,G09165,test,0704e39c73300345882ee581a802e5a7,cc2e4f1a4c2c6ceb
Lung_Opacity-776.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-776.png,Lung_Opacity/masks/Lung_Opacity-776.png,True,G09166,train,d126b50aee17d7da5fd6b1ac2470d5c7,d02b5f085c1b5b5b
Lung_Opacity-777.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-777.png,Lung_Opacity/masks/Lung_Opacity-777.png,True,G09167,val,872c105ec564161ead3a8949bc2b6472,dc3d42434f0f0f2d
Lung_Opacity-778.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-778.png,Lung_Opacity/masks/Lung_Opacity-778.png,True,G09168,test,1c5881fb430f559517b83195d28a10cb,883fd20b1c5b4f1e
Lung_Opacity-779.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-779.png,Lung_Opacity/masks/Lung_Opacity-779.png,True,G09169,test,b8e419c2452a5aec39b051f5d39113a7,91281b5b7b7980bb
Lung_Opacity-78.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-78.png,Lung_Opacity/masks/Lung_Opacity-78.png,True,G09170,train,591efe37a25134db46dd75478568ebf9,cd2d509f1e4a4d1d
Lung_Opacity-780.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-780.png,Lung_Opacity/masks/Lung_Opacity-780.png,True,G09171,train,cd36ae416f07a3657959896fcb42ccfc,d92906561e6a6b75
Lung_Opacity-781.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-781.png,Lung_Opacity/masks/Lung_Opacity-781.png,True,G09172,train,8b31a032d8ad0accfc969a848ed6e1c8,d47ca7d397415518
Lung_Opacity-782.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-782.png,Lung_Opacity/masks/Lung_Opacity-782.png,True,G09173,train,ddd1d735d6c167008bfda130807823d1,896f465859c749e6
Lung_Opacity-783.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-783.png,Lung_Opacity/masks/Lung_Opacity-783.png,True,G09174,train,1347ff108f0855375358d300e45ed8b1,917f46141f151f1d
Lung_Opacity-784.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-784.png,Lung_Opacity/masks/Lung_Opacity-784.png,True,G09175,train,938e335c976bd4c3282dc6663fc5e021,817d0f0e1e1e1d1f
Lung_Opacity-785.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-785.png,Lung_Opacity/masks/Lung_Opacity-785.png,True,G09176,train,5c17c1b89cd18b7f94175b7b6397395a,dd2c52c34a1f8b4b
Lung_Opacity-786.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-786.png,Lung_Opacity/masks/Lung_Opacity-786.png,True,G09177,test,cca5c9658922eb8791ac4e0f2f01fd2b,8d4f075e696d7160
Lung_Opacity-787.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-787.png,Lung_Opacity/masks/Lung_Opacity-787.png,True,G09178,train,9650f28de1b1bbd0f75aaa022299fc12,99690e13194e3ef9
Lung_Opacity-788.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-788.png,Lung_Opacity/masks/Lung_Opacity-788.png,True,G09179,train,a7a3a51e4c5b05b1466c057f1e477a92,d42f8515170f0f4f
Lung_Opacity-789.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-789.png,Lung_Opacity/masks/Lung_Opacity-789.png,True,G09180,train,b13ebe6bda0ce87796882e1f948564c9,8c4d471b1d2d3b2d
Lung_Opacity-79.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-79.png,Lung_Opacity/masks/Lung_Opacity-79.png,True,G09181,train,b3399a22166b32d2ee94163a4a82950f,85680e0f5d4f4f4d
Lung_Opacity-790.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-790.png,Lung_Opacity/masks/Lung_Opacity-790.png,True,G09182,train,998149658d457df71bfe5c83399cf045,d17f57025a5e5a0a
Lung_Opacity-791.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-791.png,Lung_Opacity/masks/Lung_Opacity-791.png,True,G09183,val,40bef8c607177f158f26c8644e7dcdb0,d179421a39633b3e
Lung_Opacity-792.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-792.png,Lung_Opacity/masks/Lung_Opacity-792.png,True,G09184,train,1ef335bcf39e7af528d19b012ffa593a,9d69ac1af919e930
Lung_Opacity-793.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-793.png,Lung_Opacity/masks/Lung_Opacity-793.png,True,G09185,train,e9a1934afb00c551858d3d089212eb95,917b061e797969e0
Lung_Opacity-794.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-794.png,Lung_Opacity/masks/Lung_Opacity-794.png,True,G09186,val,6a4ba5e5336b8be384c59d8faf39f86b,8129c6171d4b7b7c
Lung_Opacity-795.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-795.png,Lung_Opacity/masks/Lung_Opacity-795.png,True,G09187,val,4ece05f7807e15029ad9689af7ed9f0c,812f869e9fa3f1a0
Lung_Opacity-796.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-796.png,Lung_Opacity/masks/Lung_Opacity-796.png,True,G09188,test,f1add9e9fcdbd8ecdfbca08608aed8d4,d5204b49642f2f6f
Lung_Opacity-797.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-797.png,Lung_Opacity/masks/Lung_Opacity-797.png,True,G09189,test,26c3db8ee1a9c72907ce8da432f56e01,d42f470e68e5c2d3
Lung_Opacity-798.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-798.png,Lung_Opacity/masks/Lung_Opacity-798.png,True,G09190,test,a8b5c881af643c70ac150b52bc3cb7dd,91604e5e1b5f1a5d
Lung_Opacity-799.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-799.png,Lung_Opacity/masks/Lung_Opacity-799.png,True,G09191,train,18185af020cb7b625b18b86fd3faba9b,d17b0586561c5e5d
Lung_Opacity-8.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-8.png,Lung_Opacity/masks/Lung_Opacity-8.png,True,G09192,train,ed1801943976b91140cfa3ddd5e6e568,c93f96c2074a3d59
Lung_Opacity-80.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-80.png,Lung_Opacity/masks/Lung_Opacity-80.png,True,G09193,train,101c94c2e9867abef84753576d6e5e12,9c79434bd15b414f
Lung_Opacity-800.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-800.png,Lung_Opacity/masks/Lung_Opacity-800.png,True,G09194,train,751e7b649002b45bb708fdf531191dac,942e07585f5c7c69
Lung_Opacity-801.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-801.png,Lung_Opacity/masks/Lung_Opacity-801.png,True,G09195,train,e2eba8c9f4c282f65d0dd5e426ee1b76,94784f861e5a6979
Lung_Opacity-802.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-802.png,Lung_Opacity/masks/Lung_Opacity-802.png,True,G09196,train,5164ce3a4989620b098d4c5b1904e27c,dd2cd25bc0535353
Lung_Opacity-803.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-803.png,Lung_Opacity/masks/Lung_Opacity-803.png,True,G09197,train,5ff85160a0eef8af6bf1757237e61602,95601c9c1f1d7d1d
Lung_Opacity-804.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-804.png,Lung_Opacity/masks/Lung_Opacity-804.png,True,G09198,train,3ef5d3a2a660180f3a3e1560670d2e2b,887f82d25e44c75d
Lung_Opacity-805.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-805.png,Lung_Opacity/masks/Lung_Opacity-805.png,True,G09199,train,1f27efe179dc8428acfec183a7b894ae,897c0459396f0afe
Lung_Opacity-806.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-806.png,Lung_Opacity/masks/Lung_Opacity-806.png,True,G09200,train,632cefbacd0f147c33edbf659531cc57,8c2f070c6d4f4afa
Lung_Opacity-807.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-807.png,Lung_Opacity/masks/Lung_Opacity-807.png,True,G09201,train,f9f8bd12fb2ef969c6b464bd1715948a,c52e165a5c6e6ce8
Lung_Opacity-808.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-808.png,Lung_Opacity/masks/Lung_Opacity-808.png,True,G09202,val,f4631b6fb621adf658462a910c1d7653,d42ec31b0ded2c3c
Lung_Opacity-809.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-809.png,Lung_Opacity/masks/Lung_Opacity-809.png,True,G09203,val,6d153b1e1797076ed119c076848063a0,996896471a4f697a
Lung_Opacity-81.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-81.png,Lung_Opacity/masks/Lung_Opacity-81.png,True,G09204,train,da129daadbb1aa06188c38c3b199056d,d81a1a1a74e2e6b7
Lung_Opacity-810.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-810.png,Lung_Opacity/masks/Lung_Opacity-810.png,True,G09205,train,1cbad3378b6bbc10b20e81d8578ccc94,c93b161a191b5b5d
Lung_Opacity-811.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-811.png,Lung_Opacity/masks/Lung_Opacity-811.png,True,G09206,train,e457653975354982d72778706e987af1,d53b4a865a1b7929
Lung_Opacity-812.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-812.png,Lung_Opacity/masks/Lung_Opacity-812.png,True,G09207,val,233dc864e328172af9c7841aa59774b2,8778ac768b899972
Lung_Opacity-813.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-813.png,Lung_Opacity/masks/Lung_Opacity-813.png,True,G09208,test,27de908ae575256c389acbb444b78bcd,9c3cc3d31a4d4d3c
Lung_Opacity-814.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-814.png,Lung_Opacity/masks/Lung_Opacity-814.png,True,G09209,test,37e4d9345875e89ed894f679fe6b2cf5,d87b520a1b1b2a3f
Lung_Opacity-815.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-815.png,Lung_Opacity/masks/Lung_Opacity-815.png,True,G09210,test,f9c0cd5b90644238e6d562847d3115ea,d43c43c01feca5b5
Lung_Opacity-816.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-816.png,Lung_Opacity/masks/Lung_Opacity-816.png,True,G09211,train,3e214193ed039ab89f1954c17cd4899c,f945d61a4d1b194e
Lung_Opacity-817.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-817.png,Lung_Opacity/masks/Lung_Opacity-817.png,True,G09212,train,05b1c8fc0b4ba4f3f4f37a8d52243fd3,9131b6ba392b3a3a
Lung_Opacity-818.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-818.png,Lung_Opacity/masks/Lung_Opacity-818.png,True,G09213,train,a5e94f079f35d41efa44401bfe5e0942,c3f031a1d36f855a
Lung_Opacity-819.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-819.png,Lung_Opacity/masks/Lung_Opacity-819.png,True,G09214,train,836e296b3ce79c347ab3ad19d362596b,da3d5b135b0d0b0e
Lung_Opacity-82.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-82.png,Lung_Opacity/masks/Lung_Opacity-82.png,True,G09215,train,6752b0ac68b97b12453465a03538c774,cf2bd54d4f0b4909
Lung_Opacity-820.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-820.png,Lung_Opacity/masks/Lung_Opacity-820.png,True,G09216,train,05c76b522f02acabd1a472d3225ab121,95784e1e5a69ec2c
Lung_Opacity-821.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-821.png,Lung_Opacity/masks/Lung_Opacity-821.png,True,G09217,train,ada1123f7faf149c82b35fb100a3cee6,dd3cd20bd54e1d44
Lung_Opacity-822.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-822.png,Lung_Opacity/masks/Lung_Opacity-822.png,True,G09218,train,bf6e857753da707a7b85316a5c7f8c8f,d8fd530a4a5d0f0e
Lung_Opacity-823.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-823.png,Lung_Opacity/masks/Lung_Opacity-823.png,True,G09219,train,ed6f59a0524195d848aad8dfdc4f335b,816f0e4b4f4f4fc0
Lung_Opacity-824.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-824.png,Lung_Opacity/masks/Lung_Opacity-824.png,True,G09220,train,0ac64b4c674be2536627cd82ce7c8d9e,917bb7561a5a1a19
Lung_Opacity-825.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-825.png,Lung_Opacity/masks/Lung_Opacity-825.png,True,G09221,train,0cf0a3babc321561a90619942e7a1c04,907d28195f7878ea
Lung_Opacity-826.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-826.png,Lung_Opacity/masks/Lung_Opacity-826.png,True,G09222,train,5f76178ba04df5bf456e0bb4ea5407c9,d43b570b0d0d2f2e
Lung_Opacity-827.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-827.png,Lung_Opacity/masks/Lung_Opacity-827.png,True,G09223,train,debed6596d9ddbee4c0229ba7b789d39,956315343333b733
Lung_Opacity-828.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-828.png,Lung_Opacity/masks/Lung_Opacity-828.png,True,G09224,val,2cb3aad81fc6357a37b28686fa55bfc4,c33cc387cc3d32c9
Lung_Opacity-829.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-829.png,Lung_Opacity/masks/Lung_Opacity-829.png,True,G09225,val,9458caf58361e93fde93cbb2822e01d3,931b1c1a5b5e5c5c
Lung_Opacity-83.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-83.png,Lung_Opacity/masks/Lung_Opacity-83.png,True,G09226,test,f4929643e09a16d5c31b9d4520ea446e,983b531a39293da7
Lung_Opacity-830.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-830.png,Lung_Opacity/masks/Lung_Opacity-830.png,True,G09227,val,9381adc814e37d4448f56e0b27943480,c82f154a1d595bb5
Lung_Opacity-831.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-831.png,Lung_Opacity/masks/Lung_Opacity-831.png,True,G09228,test,4fb6afe400cc4eef0245104803e965e6,c13fc2381d1f3c1b
Lung_Opacity-832.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-832.png,Lung_Opacity/masks/Lung_Opacity-832.png,True,G09229,train,9b9f9a4edd4778751f58fa95f1785725,897f06161a4b5e79
Lung_Opacity-833.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-833.png,Lung_Opacity/masks/Lung_Opacity-833.png,True,G09230,test,e1b28637bf489eebbe4da7466acbd4ac,813bd6531c59595d
Lung_Opacity-834.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-834.png,Lung_Opacity/masks/Lung_Opacity-834.png,True,G09231,train,f4d921c119c6da043888856d7b6fc80f,c87b05161f997ad8
Lung_Opacity-835.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-835.png,Lung_Opacity/masks/Lung_Opacity-835.png,True,G09232,train,7b57029727ec955b07fe641df809a74e,956a1f590a1b5d59
Lung_Opacity-836.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-836.png,Lung_Opacity/masks/Lung_Opacity-836.png,True,G09233,train,86d6c449e17ec5934441fd3baf20ace8,9c7e47180d4a3e4f
Lung_Opacity-837.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-837.png,Lung_Opacity/masks/Lung_Opacity-837.png,True,G09234,train,e1d577f8c0b99c329c8679545b9bdd71,d93d461b1d4b3958
Lung_Opacity-838.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-838.png,Lung_Opacity/masks/Lung_Opacity-838.png,True,G09235,train,9bbe29773707f9851e1b4fe924f50e72,d03d414f6c4f1e5a
Lung_Opacity-839.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-839.png,Lung_Opacity/masks/Lung_Opacity-839.png,True,G09236,train,7b89006b82de3210a74876e00366af79,857a1c5f7d68e4a0
Lung_Opacity-84.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-84.png,Lung_Opacity/masks/Lung_Opacity-84.png,True,G09237,train,36bc301383c0e7904430139e91070e23,c939d25d1c2931af
Lung_Opacity-840.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-840.png,Lung_Opacity/masks/Lung_Opacity-840.png,True,G09238,train,8a21d82ac925ddc4c48790e12fc73940,857c0e47686ac7c7
Lung_Opacity-841.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-841.png,Lung_Opacity/masks/Lung_Opacity-841.png,True,G09239,train,40c240e9dc0c0692a32ffc73525812eb,837b05561b1b5be1
Lung_Opacity-842.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-842.png,Lung_Opacity/masks/Lung_Opacity-842.png,True,G09240,train,eb4f45cc89551ee49af14a8bfbcba068,c17b56061a4b5d5b
Lung_Opacity-843.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-843.png,Lung_Opacity/masks/Lung_Opacity-843.png,True,G09241,train,9424a4ea25fa67eb46d0a3a9065f7909,d039d21a1d1a3f3e
Lung_Opacity-844.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-844.png,Lung_Opacity/masks/Lung_Opacity-844.png,True,G09242,train,e6c1594dc4cebe50b895100a7f26210b,913a071b195b1d7e
Lung_Opacity-845.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-845.png,Lung_Opacity/masks/Lung_Opacity-845.png,True,G09243,train,789a834c53c022d2a04c03537dcb9acb,9969c7560e1c495f
Lung_Opacity-846.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-846.png,Lung_Opacity/masks/Lung_Opacity-846.png,True,G09244,train,600966008df6f7dba7fad846008064cd,d42f434e2e2275f4
Lung_Opacity-847.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-847.png,Lung_Opacity/masks/Lung_Opacity-847.png,True,G09245,train,aa10fd961e8fdc51e1fa7a964ef42fc8,914b4c5e59791979
Lung_Opacity-848.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-848.png,Lung_Opacity/masks/Lung_Opacity-848.png,True,G09246,val,39cce560c2fb0035b14486c6ab1c772c,d838d7161e1a0b3f
Lung_Opacity-849.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-849.png,Lung_Opacity/masks/Lung_Opacity-849.png,True,G09247,val,8ffb8c8ad35172407a079b61483a5f84,d12b1e1e0d1f3939
Lung_Opacity-85.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-85.png,Lung_Opacity/masks/Lung_Opacity-85.png,True,G09248,val,4e1146445ae9704c6097e80cc4bee64b,dc2f525f584f1d04
Lung_Opacity-850.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-850.png,Lung_Opacity/masks/Lung_Opacity-850.png,True,G09249,test,830e6443634d2a6fb37cfcd4a72fb9a4,9168465a595b7957
Lung_Opacity-851.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-851.png,Lung_Opacity/masks/Lung_Opacity-851.png,True,G09250,test,e4cad295bb51bc6a7fa35f77058808a0,85a8a41e1f8f1e1f
Lung_Opacity-852.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-852.png,Lung_Opacity/masks/Lung_Opacity-852.png,True,G09251,test,5a9b57ebe6ef4dda335f1f9cc90f8a6d,cc2d574b4b1b0c4f
Lung_Opacity-853.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-853.png,Lung_Opacity/masks/Lung_Opacity-853.png,True,G09252,train,fcb2da50f00487708349b7fc815899f4,846fc7070d4d6d69
Lung_Opacity-854.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-854.png,Lung_Opacity/masks/Lung_Opacity-854.png,True,G09253,train,6acc3b6fd356dc9b2ba098d1e2335a99,9168861a1b5d795f
Lung_Opacity-855.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-855.png,Lung_Opacity/masks/Lung_Opacity-855.png,True,G09254,train,e0c798156a524ad7e195abf4149a2e5c,956a2c879b1a62ed
Lung_Opacity-856.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-856.png,Lung_Opacity/masks/Lung_Opacity-856.png,True,G09255,train,44d0859b92d57c1e3e48e711b58522da,897d961a4a5d784d
Lung_Opacity-857.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-857.png,Lung_Opacity/masks/Lung_Opacity-857.png,True,G09256,train,00154d68149a2cb081c1356ca9532056,916fc79e0ccb8a58
Lung_Opacity-858.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-858.png,Lung_Opacity/masks/Lung_Opacity-858.png,True,G09257,train,09d59c4b8176dd5f3c43f6ea2656cab6,c462414d6c6efaba
Lung_Opacity-859.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-859.png,Lung_Opacity/masks/Lung_Opacity-859.png,True,G09258,train,ffd6a8e62e1f172d0dea3c0d98236c5c,c02d065f0f1e3f4d
Lung_Opacity-86.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-86.png,Lung_Opacity/masks/Lung_Opacity-86.png,True,G09259,train,526926153004949fc65cc31e4a3cdd91,c97d56d2554c5459
Lung_Opacity-860.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-860.png,Lung_Opacity/masks/Lung_Opacity-860.png,True,G09260,train,ee44d8acf39c87a3dc418c296c088b63,9139521e8bb9bca5
Lung_Opacity-861.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-861.png,Lung_Opacity/masks/Lung_Opacity-861.png,True,G09261,train,c789a52fbd621afe73df60ea537e4cdc,d52a0c5d5d5c6ce8
Lung_Opacity-862.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-862.png,Lung_Opacity/masks/Lung_Opacity-862.png,True,G09262,train,fa1af80b707b6fed6edd13c6d5cf419d,9c4f477c68f1a330
Lung_Opacity-863.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-863.png,Lung_Opacity/masks/Lung_Opacity-863.png,True,G09263,train,b0651df3c17e6e8b8ef1988a5e13d0c8,d52a76094d555e4e
Lung_Opacity-864.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-864.png,Lung_Opacity/masks/Lung_Opacity-864.png,True,G09264,val,0da753875c873307a134be24706ba7b4,d43d6fc238f031b1
Lung_Opacity-865.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-865.png,Lung_Opacity/masks/Lung_Opacity-865.png,True,G09265,train,7c610a7879f4e9a2b4e456525e52de4d,c13916065e59f8f9
Lung_Opacity-866.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-866.png,Lung_Opacity/masks/Lung_Opacity-866.png,True,G09266,test,b8a85f1a9d6b9f227d60954f32bc3f89,917b8f190c195b5b
Lung_Opacity-867.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-867.png,Lung_Opacity/masks/Lung_Opacity-867.png,True,G09267,val,53e710096f20fd6bf358eb77d010ad43,8138169e5b7b5879
Lung_Opacity-868.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-868.png,Lung_Opacity/masks/Lung_Opacity-868.png,True,G09268,val,7f2ab6a669b8e5b776521da402d89759,903b43574e465e5e
Lung_Opacity-869.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-869.png,Lung_Opacity/masks/Lung_Opacity-869.png,True,G09269,test,60aac29a21dd1e9e6beec6cc632d4b5e,857c164a0f1d7c7c
Lung_Opacity-87.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-87.png,Lung_Opacity/masks/Lung_Opacity-87.png,True,G09270,train,5dec6fc58da07687f471effe626877ff,d13bd7125b495a1a
Lung_Opacity-870.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-870.png,Lung_Opacity/masks/Lung_Opacity-870.png,True,G09271,test,742234e8c96466d05abd1aae4865a7fd,c02f5a0b3dbdb494
Lung_Opacity-871.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-871.png,Lung_Opacity/masks/Lung_Opacity-871.png,True,G09272,train,4310e13ee0732ab2f1b2779075ad4bbc,9c2c4543195f4f5d
Lung_Opacity-872.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-872.png,Lung_Opacity/masks/Lung_Opacity-872.png,True,G09273,train,ed6da70e01f19fde4342160aa393ef7f,dd2e435b0c4d0c6f
Lung_Opacity-873.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-873.png,Lung_Opacity/masks/Lung_Opacity-873.png,True,G09274,train,bfae9672c803fb1b6d2d8bf7bbfebe8b,956a9788653aa7d8
Lung_Opacity-874.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-874.png,Lung_Opacity/masks/Lung_Opacity-874.png,True,G09275,train,94d8c6035a3d6b9a8b724d454f5941f0,c42f054a2a6eaef6
Lung_Opacity-875.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-875.png,Lung_Opacity/masks/Lung_Opacity-875.png,True,G09276,train,b7bd201825463c892fa775d9be677931,82591d725a6b63cb
Lung_Opacity-876.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-876.png,Lung_Opacity/masks/Lung_Opacity-876.png,True,G09277,train,bc4fa9ab3f3d88154df57c1b5c7a9e5f,dc35d74f1c41492e
Lung_Opacity-877.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-877.png,Lung_Opacity/masks/Lung_Opacity-877.png,True,G09278,train,3aca95473b828dad053dd8c1af8eed37,d42e87531f462e2e
Lung_Opacity-878.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-878.png,Lung_Opacity/masks/Lung_Opacity-878.png,True,G09279,train,e30cdc7a86c2c25d4c4867f0cb9801a9,993f94561f545a0b
Lung_Opacity-879.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-879.png,Lung_Opacity/masks/Lung_Opacity-879.png,True,G09280,val,3e6231a95ef1fca53595c094d4bbcd2b,c42e3132f486cf6d
Lung_Opacity-88.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-88.png,Lung_Opacity/masks/Lung_Opacity-88.png,True,G09281,test,16de1974f16286a8d73a8eaadff068f6,817b168e1f79e845
Lung_Opacity-880.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-880.png,Lung_Opacity/masks/Lung_Opacity-880.png,True,G09282,train,d3f3495528afc27f313bc03ade970f98,ca2b7751ab52b50a
Lung_Opacity-881.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-881.png,Lung_Opacity/masks/Lung_Opacity-881.png,True,G09283,train,145b8735243a703c13fc63e25a201b16,d97dd75610494d49
Lung_Opacity-882.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-882.png,Lung_Opacity/masks/Lung_Opacity-882.png,True,G09284,train,6f268158adf6108f96b757f4f94ed68d,c03d4b451a2f6d3d
Lung_Opacity-883.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-883.png,Lung_Opacity/masks/Lung_Opacity-883.png,True,G09285,train,eea9cf5348a5428ad890c090b321851a,907ed61e183d381f
Lung_Opacity-884.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-884.png,Lung_Opacity/masks/Lung_Opacity-884.png,True,G09286,val,5dc97fef9391d80725499ae54c2ac6a3,9d3dc79e4b411cc8
Lung_Opacity-885.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-885.png,Lung_Opacity/masks/Lung_Opacity-885.png,True,G09287,train,6ffe31b2fecd744a5414f0f1e0862238,956687d70aef6460
Lung_Opacity-886.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-886.png,Lung_Opacity/masks/Lung_Opacity-886.png,True,G09288,val,3a2a0a5aa5059e9b526be8ff31d77fbb,dd28074a0f4f7c0f
Lung_Opacity-887.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-887.png,Lung_Opacity/masks/Lung_Opacity-887.png,True,G09289,test,9222d642cfb6dddbef53085087f993a5,943c43c30f6efd28
Lung_Opacity-888.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-888.png,Lung_Opacity/masks/Lung_Opacity-888.png,True,G09290,test,b92d0dd98cd7621b3e22142686019552,d02f521f0f0b5d3c
Lung_Opacity-889.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-889.png,Lung_Opacity/masks/Lung_Opacity-889.png,True,G09291,train,45627046c59603cb23f023822561f8f3,9919d64a18bea3b9
Lung_Opacity-89.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-89.png,Lung_Opacity/masks/Lung_Opacity-89.png,True,G09292,train,405ae6531597cda68bc57412f2beb68a,d42ac3031f0f0b7f
Lung_Opacity-890.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-890.png,Lung_Opacity/masks/Lung_Opacity-890.png,True,G09293,train,85f783c0d68e3c3804d341040889d309,81ebc615011adf5f
Lung_Opacity-891.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-891.png,Lung_Opacity/masks/Lung_Opacity-891.png,True,G09294,train,e79b27de04baaeb7bc18c905215cffe9,d12a5e4d48087f3f
Lung_Opacity-892.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-892.png,Lung_Opacity/masks/Lung_Opacity-892.png,True,G09295,train,1f48eed5de12184270a4e615a78101c2,916f0e1e1b59497c
Lung_Opacity-893.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-893.png,Lung_Opacity/masks/Lung_Opacity-893.png,True,G09296,train,cef75a233e5121a9fd224d6022c7609b,d03f4353194f495d
Lung_Opacity-894.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-894.png,Lung_Opacity/masks/Lung_Opacity-894.png,True,G09297,val,09a6d8996069f9b3baa6a3ceea12538e,817c0c5d5d5d5a5a
Lung_Opacity-895.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-895.png,Lung_Opacity/masks/Lung_Opacity-895.png,True,G09298,train,d03c63cd2c244666369f942cf1af47c3,ca2e120b4b4f4e6f
Lung_Opacity-896.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-896.png,Lung_Opacity/masks/Lung_Opacity-896.png,True,G09299,train,ce4dde41bc40ed7a56fe3114faddc4ae,c42d424b6f2cf4f4
Lung_Opacity-897.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-897.png,Lung_Opacity/masks/Lung_Opacity-897.png,True,G09300,train,f7239d26f5d5e2f41c79e127431d8a23,c075f21b0ac34b5f
Lung_Opacity-898.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-898.png,Lung_Opacity/masks/Lung_Opacity-898.png,True,G09301,train,9e8955e4517ce3dffaa576ad47fa44df,807b9716075b1b5b
Lung_Opacity-899.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-899.png,Lung_Opacity/masks/Lung_Opacity-899.png,True,G09302,test,428a381b70fab721cbff718cc3e9b6ef,95680e571a5b5b59
Lung_Opacity-9.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-9.png,Lung_Opacity/masks/Lung_Opacity-9.png,True,G09303,train,b536a8be6a1648f00c0cd3f4305213b1,d128565c487d78f5
Lung_Opacity-90.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-90.png,Lung_Opacity/masks/Lung_Opacity-90.png,True,G09304,test,63245bc0e041f37f90f98e314bea18f9,91794e16181fed1b
Lung_Opacity-900.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-900.png,Lung_Opacity/masks/Lung_Opacity-900.png,True,G09305,train,e5d4a7dc573175950cb2415f58424d5c,d02bd713da4a4a7a
Lung_Opacity-901.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-901.png,Lung_Opacity/masks/Lung_Opacity-901.png,True,G09306,train,21ef66f71e1de69b95d4ce9b1515277f,d42f42550f4ee50f
Lung_Opacity-902.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-902.png,Lung_Opacity/masks/Lung_Opacity-902.png,True,G09307,train,c57b19586d9540ac7e6125b2ab094a9e,807fe585b098cec7
Lung_Opacity-903.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-903.png,Lung_Opacity/masks/Lung_Opacity-903.png,True,G09308,val,a1917e6ddbcdeaecc0bc020dc11db2e3,dd7cc64317c51a61
Lung_Opacity-904.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-904.png,Lung_Opacity/masks/Lung_Opacity-904.png,True,G09309,val,c806d26643afd88b2ce346abe39e21d0,8d2a02434b4f5f7e
Lung_Opacity-905.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-905.png,Lung_Opacity/masks/Lung_Opacity-905.png,True,G09310,train,4d37d9826f2134be73dc59138c3d4c6b,91781f7b6b6261e0
Lung_Opacity-906.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-906.png,Lung_Opacity/masks/Lung_Opacity-906.png,True,G09311,test,1855f6f7996f3999bd263b559c989519,d13b1d081d1b7b1b
Lung_Opacity-907.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-907.png,Lung_Opacity/masks/Lung_Opacity-907.png,True,G09312,train,adcd7e7846cfbcefa604ccfd99190a05,817e0f485d154d5f
Lung_Opacity-908.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-908.png,Lung_Opacity/masks/Lung_Opacity-908.png,True,G09313,train,2e91695b86a2ae1bb75e65c0a962e6af,cd6f2f60e0f0b027
Lung_Opacity-909.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-909.png,Lung_Opacity/masks/Lung_Opacity-909.png,True,G09314,train,16a479dae1d766bde7b382780ee64647,9528c456131b5f7b
Lung_Opacity-91.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-91.png,Lung_Opacity/masks/Lung_Opacity-91.png,True,G09315,train,e6536fd5d4990d2a4003e0f8a97c41c2,817f060c4f717373
Lung_Opacity-910.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-910.png,Lung_Opacity/masks/Lung_Opacity-910.png,True,G09316,train,f8e1f1a821d65634d63c1a634d2003a5,dc29431f0a999f93
Lung_Opacity-911.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-911.png,Lung_Opacity/masks/Lung_Opacity-911.png,True,G09317,train,c0b95bcbc6c8642440690d4285e137ef,942e835f0e6a5cdc
Lung_Opacity-912.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-912.png,Lung_Opacity/masks/Lung_Opacity-912.png,True,G09318,test,9eee225506da3cbbe2fd17349794eda0,8978161e195b5b7a
Lung_Opacity-913.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-913.png,Lung_Opacity/masks/Lung_Opacity-913.png,True,G09319,train,8c40e0e4b8d0467aa9c2c54355c888b7,cc2dd05b0f4d0b1f
Lung_Opacity-914.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-914.png,Lung_Opacity/masks/Lung_Opacity-914.png,True,G09320,train,68ca391ae2ecc1adfac65ad08be6e27f,917bc20b69c72bc3
Lung_Opacity-915.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-915.png,Lung_Opacity/masks/Lung_Opacity-915.png,True,G09321,val,efec8214f57ed75d3664cc7022859cd9,c03d7b195d59584e
Lung_Opacity-916.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-916.png,Lung_Opacity/masks/Lung_Opacity-916.png,True,G09322,val,918c59ff9680e41152faa842138e5b58,c42b430a2a6b2fef
Lung_Opacity-917.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-917.png,Lung_Opacity/masks/Lung_Opacity-917.png,True,G09323,train,6a7fa3f89d8855daf6a3c599b47b7ca5,cc2df2d2db4bc04a
Lung_Opacity-918.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-918.png,Lung_Opacity/masks/Lung_Opacity-918.png,True,G09324,train,ddad4890ff801b41321d10a4ad92a926,d978c61b0d4f3b44
Lung_Opacity-919.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-919.png,Lung_Opacity/masks/Lung_Opacity-919.png,True,G09325,train,a8c0dbb9c783aa496e9b1f802b4bd608,dd6d121b49ac3cf0
Lung_Opacity-92.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-92.png,Lung_Opacity/masks/Lung_Opacity-92.png,True,G09326,train,9cce77808aaeeb5fe77bd3ed1cd727a4,9968851a1f5a59cf
Lung_Opacity-920.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-920.png,Lung_Opacity/masks/Lung_Opacity-920.png,True,G09327,train,e01e8b59f039e6767980c61e2bfd437d,d52a531b2c2c3dbc
Lung_Opacity-921.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-921.png,Lung_Opacity/masks/Lung_Opacity-921.png,True,G09328,val,d9fc3238cd15c34bf88dbc2f3a2d315b,917a163b3b23e465
Lung_Opacity-922.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-922.png,Lung_Opacity/masks/Lung_Opacity-922.png,True,G09329,train,f20b7d3e00dbab7594709502e1e82b93,996a86124a5d7cfa
Lung_Opacity-923.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-923.png,Lung_Opacity/masks/Lung_Opacity-923.png,True,G09330,test,6ad757e649e3ea90e4d73faf39afac70,dd2c92d34b4dc658
Lung_Opacity-924.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-924.png,Lung_Opacity/masks/Lung_Opacity-924.png,True,G09331,test,98897752c26dbb6e281f4cca7b47627b,807bc61aba507b79
Lung_Opacity-925.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-925.png,Lung_Opacity/masks/Lung_Opacity-925.png,True,G09332,val,d02ae89b6f7953909b0a12748888f7d5,cd341713471e471f
Lung_Opacity-926.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-926.png,Lung_Opacity/masks/Lung_Opacity-926.png,True,G09333,train,f4f8d487263dbfe6e77e8ecc11e0121f,846d074b2e6e2e6e
Lung_Opacity-927.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-927.png,Lung_Opacity/masks/Lung_Opacity-927.png,True,G09334,train,6fa88bf5f9ac4b7e4c85f106f650a4ba,81791e59196b697c
Lung_Opacity-928.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-928.png,Lung_Opacity/masks/Lung_Opacity-928.png,True,G09335,train,c9cf381f6dd14c33ae77f9151d4c2923,894fc649e969e136
Lung_Opacity-929.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-929.png,Lung_Opacity/masks/Lung_Opacity-929.png,True,G09336,train,c22e90b27ee7534bad97790c4646a562,dd2e510b4c4f4e4e
Lung_Opacity-93.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-93.png,Lung_Opacity/masks/Lung_Opacity-93.png,True,G09337,train,0b2eec7f13709db13d183da99a5fed15,91b968ce17b9b164
Lung_Opacity-930.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-930.png,Lung_Opacity/masks/Lung_Opacity-930.png,True,G09338,train,e07d2b6d508609b137497eac8b3270af,c43e07430b5f1e5e
Lung_Opacity-931.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-931.png,Lung_Opacity/masks/Lung_Opacity-931.png,True,G09339,train,996d07156f671b03c4b983a059dfabc1,dc2b530f2e0c1f3c
Lung_Opacity-932.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-932.png,Lung_Opacity/masks/Lung_Opacity-932.png,True,G09340,train,b253dc14828ca455b6696712c5413ec8,d03fc1531f79091d
Lung_Opacity-933.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-933.png,Lung_Opacity/masks/Lung_Opacity-933.png,True,G09341,train,63900bf1eac2bcc9ebd0f162daf321f0,c83d4b4b4c6d2e2e
Lung_Opacity-934.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-934.png,Lung_Opacity/masks/Lung_Opacity-934.png,True,G09342,train,45cf523377876fed961ea5012cf6ee59,d43c4b430d4eae3d
Lung_Opacity-935.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-935.png,Lung_Opacity/masks/Lung_Opacity-935.png,True,G09343,train,702e1dc9b621d4e21137fc71229de2d3,cd2d1492034f5f5d
Lung_Opacity-936.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-936.png,Lung_Opacity/masks/Lung_Opacity-936.png,True,G09344,train,2a7c69bf35b133ec3b75313fa5595dd4,d52b53092933b397
Lung_Opacity-937.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-937.png,Lung_Opacity/masks/Lung_Opacity-937.png,True,G09345,train,b99dbf59c33269933a9473f5d0e069af,c56816df4b4d1c3c
Lung_Opacity-938.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-938.png,Lung_Opacity/masks/Lung_Opacity-938.png,True,G09346,train,975c5492abc4318a1d64fa2e94db66a1,dd2e52c10b4d4d4f
Lung_Opacity-939.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-939.png,Lung_Opacity/masks/Lung_Opacity-939.png,True,G09347,val,e376445c59bf469bf2a2e3b9ec495959,953a2aa55a0d9f1e
Lung_Opacity-94.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-94.png,Lung_Opacity/masks/Lung_Opacity-94.png,True,G09348,val,b8463e81548a7b2a1421af5fee62080b,817f1fd65a09194b
Lung_Opacity-940.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-940.png,Lung_Opacity/masks/Lung_Opacity-940.png,True,G09349,test,26b1f5a3baeb0e9f7c0112323cfa01f5,d52ec3094d4c2e6f
Lung_Opacity-941.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-941.png,Lung_Opacity/masks/Lung_Opacity-941.png,True,G09350,train,cb829b51b4bbc5545c7470589620da97,c56e0a1d0c4d7f4e
Lung_Opacity-942.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-942.png,Lung_Opacity/masks/Lung_Opacity-942.png,True,G09351,train,4b218a2b0395562b16a277783940e432,dd2f025f0c4c6d3c
Lung_Opacity-943.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-943.png,Lung_Opacity/masks/Lung_Opacity-943.png,True,G09352,train,04d0b8f50acec85fc1b576bd5e010307,897d87920a4dde5a
Lung_Opacity-944.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-944.png,Lung_Opacity/masks/Lung_Opacity-944.png,True,G09353,val,12265e79c2721c73143cd4094234d0a2,817a0a065f5f585f
Lung_Opacity-945.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-945.png,Lung_Opacity/masks/Lung_Opacity-945.png,True,G09354,test,8e5f36e78edd1f28cc54b44cde124bec,d42a574b5c0a4b5f
Lung_Opacity-946.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-946.png,Lung_Opacity/masks/Lung_Opacity-946.png,True,G09355,test,75ef2fe3a62fd0573c821381eeca2fdc,c83e87431b8d5c1f
Lung_Opacity-947.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-947.png,Lung_Opacity/masks/Lung_Opacity-947.png,True,G09356,train,3e7342ad79fcb69520ffaf684f4e4f19,d56cd193c4c4d29b
Lung_Opacity-948.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-948.png,Lung_Opacity/masks/Lung_Opacity-948.png,True,G09357,train,4a409baa32282cc59b58781a04c60515,91780f1f497c70f1
Lung_Opacity-949.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-949.png,Lung_Opacity/masks/Lung_Opacity-949.png,True,G09358,train,9133a73400d6f1c307978bfda03f9d68,d135bb37283a3a1a
Lung_Opacity-95.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-95.png,Lung_Opacity/masks/Lung_Opacity-95.png,True,G09359,train,618ca3d31bfc6d88a7f757a296f1dfd7,816e07185f587cee
Lung_Opacity-950.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-950.png,Lung_Opacity/masks/Lung_Opacity-950.png,True,G09360,train,04a68fdeab4e338a25cddb39cd1ad652,8129879eb89fb8b8
Lung_Opacity-951.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-951.png,Lung_Opacity/masks/Lung_Opacity-951.png,True,G09361,test,21aa7e66126734fdb53abfa060224ecd,c1791e965a5bb81a
Lung_Opacity-952.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-952.png,Lung_Opacity/masks/Lung_Opacity-952.png,True,G09362,train,e1d496c84f75de29bb5ca1f1622268da,802fff86153e832b
Lung_Opacity-953.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-953.png,Lung_Opacity/masks/Lung_Opacity-953.png,True,G09363,train,d1dad7ee9612439403046a120d086b0c,c03fd78a9c98ab07
Lung_Opacity-954.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-954.png,Lung_Opacity/masks/Lung_Opacity-954.png,True,G09364,train,7467eafb1ab983516172d3b1b0728e7a,d237d89a0a2e2e2f
Lung_Opacity-955.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-955.png,Lung_Opacity/masks/Lung_Opacity-955.png,True,G09365,train,c9f7203a88380c0a7786f8b6de4ffe66,99794787161e4e4d
Lung_Opacity-956.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-956.png,Lung_Opacity/masks/Lung_Opacity-956.png,True,G09366,test,fcf60d318a927f3330a12dec27a80094,c22f87070e6e2e2f
Lung_Opacity-957.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-957.png,Lung_Opacity/masks/Lung_Opacity-957.png,True,G09367,train,6a0637f9f275c8b4da071232578fde2a,d02d02d03fd23bde
Lung_Opacity-958.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-958.png,Lung_Opacity/masks/Lung_Opacity-958.png,True,G09368,train,38b5ba1225a5b6625545d005bd1c717b,90394f47171d8d3b
Lung_Opacity-959.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-959.png,Lung_Opacity/masks/Lung_Opacity-959.png,True,G09369,val,87b16f3f42a73c7367c46d9a09385cab,d1304b464e0fde1f
Lung_Opacity-96.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-96.png,Lung_Opacity/masks/Lung_Opacity-96.png,True,G09370,val,4a8e3b57f9eb519ef55b70e2e32e2ad2,897984c71e1ece1b
Lung_Opacity-960.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-960.png,Lung_Opacity/masks/Lung_Opacity-960.png,True,G09371,test,394a47a2469d70ae3094a30fdbf2656a,ca3f860f4e6e4643
Lung_Opacity-961.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-961.png,Lung_Opacity/masks/Lung_Opacity-961.png,True,G09372,train,db5ed13e738051532cef969e2d658d37,d07d020b1b9b1f6d
Lung_Opacity-962.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-962.png,Lung_Opacity/masks/Lung_Opacity-962.png,True,G09373,train,b17845403d92ada4857b7dcc1394b7ae,847c47831f7868f3
Lung_Opacity-963.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-963.png,Lung_Opacity/masks/Lung_Opacity-963.png,True,G09374,train,bfbc9bba85feb4af51257e40df15f7ac,c52e430f4e4d4c6f
Lung_Opacity-964.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-964.png,Lung_Opacity/masks/Lung_Opacity-964.png,True,G09375,train,ae211ba03e49898dcf19c3479e2336e4,917b431b195b18eb
Lung_Opacity-965.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-965.png,Lung_Opacity/masks/Lung_Opacity-965.png,True,G09376,train,1a46ca96e08eab9a0b945564d828ae31,d47e83050f071f1f
Lung_Opacity-966.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-966.png,Lung_Opacity/masks/Lung_Opacity-966.png,True,G09377,train,f468cbe6ba10ca074360172ac6614816,9fe8849e83bac8d8
Lung_Opacity-967.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-967.png,Lung_Opacity/masks/Lung_Opacity-967.png,True,G09378,train,4e5efe6be6bd0fd9192e4d99bb87101d,d03d43c34b0a6d6f
Lung_Opacity-968.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-968.png,Lung_Opacity/masks/Lung_Opacity-968.png,True,G09379,train,4aa10b68c3bf03c270e3ff7f7f647f38,c93e065a1cec69ad
Lung_Opacity-969.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-969.png,Lung_Opacity/masks/Lung_Opacity-969.png,True,G09380,train,e0fbea923631c94133bb3727bd9f584f,c27d540a0f7d3d0e
Lung_Opacity-97.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-97.png,Lung_Opacity/masks/Lung_Opacity-97.png,True,G09381,train,32b11f692e874401b1fb7ad41e4bcba8,cd2d02475f0d4f1e
Lung_Opacity-970.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-970.png,Lung_Opacity/masks/Lung_Opacity-970.png,True,G09382,train,e38d28916fa91590305400d2e7dd3f64,c13c1259474f79e5
Lung_Opacity-971.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-971.png,Lung_Opacity/masks/Lung_Opacity-971.png,True,G09383,train,0b73b45586ff152713e420276e081d70,9c48431f5c1ee9ba
Lung_Opacity-972.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-972.png,Lung_Opacity/masks/Lung_Opacity-972.png,True,G09384,train,852150cf34011095d2b4b1a1474e336c,813bd61e1a5a5b59
Lung_Opacity-973.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-973.png,Lung_Opacity/masks/Lung_Opacity-973.png,True,G09385,val,f8a3c3c613934b23ffcb18121b0117f6,cc2ed25d0c2b3d96
Lung_Opacity-974.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-974.png,Lung_Opacity/masks/Lung_Opacity-974.png,True,G09386,val,62e3c3124f5c4d63a06ce15602f5ee5c,816987d29e55383f
Lung_Opacity-975.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-975.png,Lung_Opacity/masks/Lung_Opacity-975.png,True,G09387,val,cb76a472325d15d596f3200765cf4785,c03d405b5a5a7b7a
Lung_Opacity-976.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-976.png,Lung_Opacity/masks/Lung_Opacity-976.png,True,G09388,test,97c3920c6ae48a4f76bab19d0b159528,9979061f5839f8a9
Lung_Opacity-977.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-977.png,Lung_Opacity/masks/Lung_Opacity-977.png,True,G09389,train,7e755dacf69f1c22f953b8cb0deb16c6,dc28d3c35f0e1a3a
Lung_Opacity-978.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-978.png,Lung_Opacity/masks/Lung_Opacity-978.png,True,G09390,train,8b1474018eab4eab0ed6f00b9ce53fb3,d52ad34b4f4d4c1c
Lung_Opacity-979.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-979.png,Lung_Opacity/masks/Lung_Opacity-979.png,True,G09391,train,f6926552409a3f19a0bdb59ad6ba9927,c128164f4f2e3b79
Lung_Opacity-98.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-98.png,Lung_Opacity/masks/Lung_Opacity-98.png,True,G09392,test,be5ba5c6989294edb6486e8b886e092d,c919161e1a393b7b
Lung_Opacity-980.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-980.png,Lung_Opacity/masks/Lung_Opacity-980.png,True,G09393,test,be6fb09bc55c0927ddefe3ea73c2142e,9978461e1b5d1b3a
Lung_Opacity-981.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-981.png,Lung_Opacity/masks/Lung_Opacity-981.png,True,G09394,train,e794d225181fc15fca1e29c7277589f4,cc3dc24a1f4919ed
Lung_Opacity-982.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-982.png,Lung_Opacity/masks/Lung_Opacity-982.png,True,G09395,train,0db8a19d191b6c236d1851d7ef341e02,907f4e5f4e18191d
Lung_Opacity-983.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-983.png,Lung_Opacity/masks/Lung_Opacity-983.png,True,G09396,train,dc2d4d87540ccb23b919a3c97a847755,c43d5b4f1e0a0d1f
Lung_Opacity-984.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-984.png,Lung_Opacity/masks/Lung_Opacity-984.png,True,G09397,train,e0d36ea10bbda2d3f302adb97d57b09a,dd63c8d08287b6b6
Lung_Opacity-985.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-985.png,Lung_Opacity/masks/Lung_Opacity-985.png,True,G09398,val,5e88e84d772a6ab0b36186041c9737fa,9b695c1d5b68a271
Lung_Opacity-986.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-986.png,Lung_Opacity/masks/Lung_Opacity-986.png,True,G09399,train,13c7ec7e9b892a76adae1497709b666e,943b531a18593f3b
Lung_Opacity-987.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-987.png,Lung_Opacity/masks/Lung_Opacity-987.png,True,G09400,test,46299363d643e4d07e606f91abd0b629,997f861e0d1e1a0f
Lung_Opacity-988.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-988.png,Lung_Opacity/masks/Lung_Opacity-988.png,True,G09401,train,ccdf0d381caaecb87d59227d8d696eae,842f034b4f4c4f6f
Lung_Opacity-989.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-989.png,Lung_Opacity/masks/Lung_Opacity-989.png,True,G09402,train,4bd767f29fd300e9317a41663845933a,d73f0e1e1f1a1e10
Lung_Opacity-99.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-99.png,Lung_Opacity/masks/Lung_Opacity-99.png,True,G09403,train,b77343958641e8ddd1c68a621a71b6fa,833c6a3c85c7d939
Lung_Opacity-990.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-990.png,Lung_Opacity/masks/Lung_Opacity-990.png,True,G09404,train,d3bec390ef02c40e455d2ba3ef928cbb,cc2dc61e4c4f4f0b
Lung_Opacity-991.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-991.png,Lung_Opacity/masks/Lung_Opacity-991.png,True,G09405,train,eea07f0579c336ccd72730182d2f52eb,dd2ac0c71ac3ad1e
Lung_Opacity-992.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-992.png,Lung_Opacity/masks/Lung_Opacity-992.png,True,G09406,train,81f8294ed0317ce933adcd7e3fe5ce4e,c42a434f4f0e4f4f
Lung_Opacity-993.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-993.png,Lung_Opacity/masks/Lung_Opacity-993.png,True,G09407,train,55da884579906c989aa746c02f1da8ab,d42d051f1d0a7d79
Lung_Opacity-994.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-994.png,Lung_Opacity/masks/Lung_Opacity-994.png,True,G09408,train,8bb565f3fdc32f4eb83753ff384d21ba,917b005a597b1dae
Lung_Opacity-995.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-995.png,Lung_Opacity/masks/Lung_Opacity-995.png,True,G09409,train,6d23e0e1417de5d01e04581a26244882,d46f474a2d2e2c2e
Lung_Opacity-996.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-996.png,Lung_Opacity/masks/Lung_Opacity-996.png,True,G09410,val,6aee3656d1e696ad51c2db762cdaffd8,856816161f1f1f1f
Lung_Opacity-997.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-997.png,Lung_Opacity/masks/Lung_Opacity-997.png,True,G09411,val,91ef6a6600ade3c298d0dc3e1f8d485b,903a575f1a0b393b
Lung_Opacity-998.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-998.png,Lung_Opacity/masks/Lung_Opacity-998.png,True,G09412,test,7cfec70e001b994eaee997494e863e51,8170d74ed81af81f
Lung_Opacity-999.png,Lung_Opacity,Lung_Opacity/images/Lung_Opacity-999.png,Lung_Opacity/masks/Lung_Opacity-999.png,True,G09413,test,f9578f6b266dfda2d27654727c22eaed,917a50163b3bb396
Normal-1.png,Normal,Normal/images/Normal-1.png,Normal/masks/Normal-1.png,True,G09414,train,973387ba59b2669d570d73dcfe55f88e,c0ac2bc27dc32f3c
Normal-10.png,Normal,Normal/images/Normal-10.png,Normal/masks/Normal-10.png,True,G09415,train,95aea838eb8d7ee0522e1abe0747129e,d979c61e1d445b19
Normal-100.png,Normal,Normal/images/Normal-100.png,Normal/masks/Normal-100.png,True,G09416,train,829b755f1aadbec50adbd1815ffe81bf,9129c21eca6f7d61
Normal-1000.png,Normal,Normal/images/Normal-1000.png,Normal/masks/Normal-1000.png,True,G09417,train,69c20913b20b88e8873d188ea3fbe58d,d05c871bcd8d4ac7
Normal-10000.png,Normal,Normal/images/Normal-10000.png,Normal/masks/Normal-10000.png,True,G09418,train,b7c38ca5ff4be275d6d616545bdf0866,d92e461a9849bbf8
Normal-10001.png,Normal,Normal/images/Normal-10001.png,Normal/masks/Normal-10001.png,True,G09419,train,d1a54f3d0429059729196b6a7e2fd262,d42c838f1c0cef2f
Normal-10002.png,Normal,Normal/images/Normal-10002.png,Normal/masks/Normal-10002.png,True,G09420,train,17d5983b6b0bd36c36ee80fac6c2a798,d42dc31d0a6e3de2
Normal-10003.png,Normal,Normal/images/Normal-10003.png,Normal/masks/Normal-10003.png,True,G09421,train,33b814032f6c80bd84d48413fffce1a9,dd7c02421b4b4f2f
Normal-10004.png,Normal,Normal/images/Normal-10004.png,Normal/masks/Normal-10004.png,True,G09422,train,d3ee948d6e8968c6778e4a1915a8c8db,817957d61e1c565a
Normal-10005.png,Normal,Normal/images/Normal-10005.png,Normal/masks/Normal-10005.png,True,G09423,val,59777c7f37e815a189296e2b829629de,dc3d5253c34b584d
Normal-10006.png,Normal,Normal/images/Normal-10006.png,Normal/masks/Normal-10006.png,True,G09424,val,409142b5e063441263029525a4906ce1,c16f069e18394daf
Normal-10007.png,Normal,Normal/images/Normal-10007.png,Normal/masks/Normal-10007.png,True,G09425,val,bcc9dcdafb05f9714254198aed1a18fd,897e0711574e1e6d
Normal-10008.png,Normal,Normal/images/Normal-10008.png,Normal/masks/Normal-10008.png,True,G09426,test,653ab2012b31b120dd082cec0e03289c,997dc51ad5471847
Normal-10009.png,Normal,Normal/images/Normal-10009.png,Normal/masks/Normal-10009.png,True,G09427,train,b5bf09e4ec68d9aaac9d0b4ae04c7e85,9d2cc21bc51a996f
Normal-1001.png,Normal,Normal/images/Normal-1001.png,Normal/masks/Normal-1001.png,True,G09428,train,c82087ec333834a44b70057660290de2,914fc758c16792c7
Normal-10010.png,Normal,Normal/images/Normal-10010.png,Normal/masks/Normal-10010.png,True,G09429,train,598b4b41602bd2701f3871d02cf4297f,c83dcf5e524f0c98
Normal-10011.png,Normal,Normal/images/Normal-10011.png,Normal/masks/Normal-10011.png,True,G09430,train,65b3da4830d1cd02422b1489e7f45126,dc3d534b4f4b4a48
Normal-10012.png,Normal,Normal/images/Normal-10012.png,Normal/masks/Normal-10012.png,True,G09431,train,65f0e48c71ae7579a72efc56b1880785,c82f430f4e0f0e6f
Normal-10013.png,Normal,Normal/images/Normal-10013.png,Normal/masks/Normal-10013.png,True,G09432,test,1017501837e6d1b887584430cc20f930,995b161a192bbbb4
Normal-10014.png,Normal,Normal/images/Normal-10014.png,Normal/masks/Normal-10014.png,True,G09433,test,8a08475dfd745070f59a50edf8f1ee60,b93dc5531a0f0d1e
Normal-10015.png,Normal,Normal/images/Normal-10015.png,Normal/masks/Normal-10015.png,True,G09434,train,f07f147de4330b357bff6147a1cfd9b1,c02f03031d5f5f5d
Normal-10016.png,Normal,Normal/images/Normal-10016.png,Normal/masks/Normal-10016.png,True,G09435,train,7ade1d2c6be5359f3747de2e78b4bb16,dd68c20f0d0a1def
Normal-10017.png,Normal,Normal/images/Normal-10017.png,Normal/masks/Normal-10017.png,True,G09436,train,8782f281974545346343e649dfa10f1a,dd28861ff8e6a891
Normal-10018.png,Normal,Normal/images/Normal-10018.png,Normal/masks/Normal-10018.png,True,G09437,train,4a1985164b4d4d3afbf56b2f96dbc996,8c0c430b4f4f5f4f
Normal-10019.png,Normal,Normal/images/Normal-10019.png,Normal/masks/Normal-10019.png,True,G09438,train,d1801bf800c126f7df4077ef1a7d1ba5,997b061a597938e9
Normal-1002.png,Normal,Normal/images/Normal-1002.png,Normal/masks/Normal-1002.png,True,G09439,train,94faeab2f1ef0866de29b3e0d9fcfd02,9829c71eda4b3d68
Normal-10020.png,Normal,Normal/images/Normal-10020.png,Normal/masks/Normal-10020.png,True,G09440,val,ee7e662c3d2f3fe9e3c2ea4d516a1b6d,d178c616594f7958
Normal-10021.png,Normal,Normal/images/Normal-10021.png,Normal/masks/Normal-10021.png,True,G09441,train,aab095a51d55a12b4db3918385288dd1,c03c4b0f0c2f3d3f
Normal-10022.png,Normal,Normal/images/Normal-10022.png,Normal/masks/Normal-10022.png,True,G09442,train,37f55d822c764b5a05d0dad8d72d0707,c13b56071a174f3b
Normal-10023.png,Normal,Normal/images/Normal-10023.png,Normal/masks/Normal-10023.png,True,G09443,train,8b9320d99309e2080b38bf3c3411e16e,987f871f08071f4b
Normal-10024.png,Normal,Normal/images/Normal-10024.png,Normal/masks/Normal-10024.png,True,G09444,train,b8781fcd48b0341db85f6d617bb71c27,dd2a50c707c60fd5
Normal-10025.png,Normal,Normal/images/Normal-10025.png,Normal/masks/Normal-10025.png,True,G09445,train,ca415fa80208d7d35397b0a13d726ef1,c02b56575b5f0d1a
Normal-10026.png,Normal,Normal/images/Normal-10026.png,Normal/masks/Normal-10026.png,True,G09446,train,3577d64897777ffa7c3dbfd61b2f0409,cc3ec25357431f4c
Normal-10027.png,Normal,Normal/images/Normal-10027.png,Normal/masks/Normal-10027.png,True,G09447,train,78c3a0d31d948e375af4a6b3f9e2d0e7,956c8719cc4e78c7
Normal-10028.png,Normal,Normal/images/Normal-10028.png,Normal/masks/Normal-10028.png,True,G09448,train,14621e177f095c8b0fc3f5f3999c8558,c83c4b0fdce6e0c3
Normal-10029.png,Normal,Normal/images/Normal-10029.png,Normal/masks/Normal-10029.png,True,G09449,val,a0c99e8a8d4567cf433dc830f3651abf,c43dc34b4c4a6e3e
Normal-1003.png,Normal,Normal/images/Normal-1003.png,Normal/masks/Normal-1003.png,True,G09450,val,6ef910faa8acaa01c6eaa54195228f7e,c12dc63aec693d64
Normal-10030.png,Normal,Normal/images/Normal-10030.png,Normal/masks/Normal-10030.png,True,G09451,test,b21049e606f8293c11292c3d4b2a18d7,d82d46188b6cb9ed
Normal-10031.png,Normal,Normal/images/Normal-10031.png,Normal/masks/Normal-10031.png,True,G09452,test,c111b3d9c8fa3e6af130a8c611b65017,d979c61a194d4b39
Normal-10032.png,Normal,Normal/images/Normal-10032.png,Normal/masks/Normal-10032.png,True,G09453,test,ae2e2b2fdeab6f78fc51ce9351e3993e,d939c21f1f471b18
Normal-10033.png,Normal,Normal/images/Normal-10033.png,Normal/masks/Normal-10033.png,True,G09454,train,ba5b56765b7e6d534df618dae12a6b12,c13946111bee2dbd
Normal-10034.png,Normal,Normal/images/Normal-10034.png,Normal/masks/Normal-10034.png,True,G09455,train,907ea7ffec23fa332de3971a0916a8f3,9978161a5e1f195b
Normal-10035.png,Normal,Normal/images/Normal-10035.png,Normal/masks/Normal-10035.png,True,G09456,train,c127356d3224fa7dad93db6883e50767,996f061e171a5c5b
Normal-10036.png,Normal,Normal/images/Normal-10036.png,Normal/masks/Normal-10036.png,True,G09457,train,28cf9b1d83492e17ee4098e8cccf59e4,dc6e421b424f5a4f
Normal-10037.png,Normal,Normal/images/Normal-10037.png,Normal/masks/Normal-10037.png,True,G09458,train,353a5b5a1e27629b106df2b6cd6cbcd3,dd2d42531bc86e3c
Normal-10038.png,Normal,Normal/images/Normal-10038.png,Normal/masks/Normal-10038.png,True,G09459,train,34d118598c4eaa525f33e26d6574ec44,f96cc65b554f1848
Normal-10039.png,Normal,Normal/images/Normal-10039.png,Normal/masks/Normal-10039.png,True,G09460,train,6dc5f779cce054f89f6d1ce5751438cc,8939871e1a555f39
Normal-1004.png,Normal,Normal/images/Normal-1004.png,Normal/masks/Normal-1004.png,True,G09461,val,e6b455e617101fd7b5a35a67c0df0a53,91fc861abc6d39a4
Normal-10040.png,Normal,Normal/images/Normal-10040.png,Normal/masks/Normal-10040.png,True,G09462,train,4e3ce437415f918876812a7dce15fc33,942c6bdb7aa4a46c
Normal-10041.png,Normal,Normal/images/Normal-10041.png,Normal/masks/Normal-10041.png,True,G09463,train,f13fcee789f82bec1072bb4d2ecbb92e,9178461a99efa8b9
Normal-10042.png,Normal,Normal/images/Normal-10042.png,Normal/masks/Normal-10042.png,True,G09464,train,cfd738f3624de42c67f7ba6b7a3e484a,987d871e0b4f1a4d
Normal-10043.png,Normal,Normal/images/Normal-10043.png,Normal/masks/Normal-10043.png,True,G09465,val,832c441efb0795689bcd946c20458bbc,d92e065a7949f8c5
Normal-10044.png,Normal,Normal/images/Normal-10044.png,Normal/masks/Normal-10044.png,True,G09466,train,db6da480dcdce1ec8b2818ad879e1787,917b57861a955b19
Normal-10045.png,Normal,Normal/images/Normal-10045.png,Normal/masks/Normal-10045.png,True,G09467,val,7e69b0815ea05a96b60d7f517147acec,9179c6061b15c75f
Normal-10046.png,Normal,Normal/images/Normal-10046.png,Normal/masks/Normal-10046.png,True,G09468,train,3239ce2dfd4a7aae9270a4ebb14f6554,bd2dc6c31b41531e
Normal-10047.png,Normal,Normal/images/Normal-10047.png,Normal/masks/Normal-10047.png,True,G09469,test,331658c1a9d327fbd00ca52c3acc4c53,8c52430f0c0fafbf
Normal-10048.png,Normal,Normal/images/Normal-10048.png,Normal/masks/Normal-10048.png,True,G09470,val,f6f00cde6e70a8127c390f676d0bc030,997f861a861a1f47
Normal-10049.png,Normal,Normal/images/Normal-10049.png,Normal/masks/Normal-10049.png,True,G09471,test,9b1951ed68daef083148cb5bf0737ee5,817a171e185de9e9
Normal-1005.png,Normal,Normal/images/Normal-1005.png,Normal/masks/Normal-1005.png,True,G09472,train,ab0405d5ca8ab16d01d0fcb981f9203b,913ec21a1d5d1dd6
Normal-10050.png,Normal,Normal/images/Normal-10050.png,Normal/masks/Normal-10050.png,True,G09473,train,c5151b4a9f58af6e335f833e940385d5,916a161e1f5a5a7a
Normal-10051.png,Normal,Normal/images/Normal-10051.png,Normal/masks/Normal-10051.png,True,G09474,test,6265926ee96dece60f9fecdb1c976c02,dd39424f1b286d39
Normal-10052.png,Normal,Normal/images/Normal-10052.png,Normal/masks/Normal-10052.png,True,G09475,train,9865d6601388b1d581055c632ed7fdd1,8168de565819cd5f
Normal-10053.png,Normal,Normal/images/Normal-10053.png,Normal/masks/Normal-10053.png,True,G09476,train,a7339ce9fee92df9158fd9e3ceabecc6,dd6c871b1d4e1a51
Normal-10054.png,Normal,Normal/images/Normal-10054.png,Normal/masks/Normal-10054.png,True,G09477,train,923ec834889d4554d84d208c21eedff0,c8795587169d5e94
Normal-10055.png,Normal,Normal/images/Normal-10055.png,Normal/masks/Normal-10055.png,True,G09478,train,1c75addd9ec1de1f1f9d245631a049e4,99699e1a19c5e4e3
Normal-10056.png,Normal,Normal/images/Normal-10056.png,Normal/masks/Normal-10056.png,True,G09479,train,0bac8f0098f19533a61d51b6e816649f,893b161a595b78cb
Normal-10057.png,Normal,Normal/images/Normal-10057.png,Normal/masks/Normal-10057.png,True,G09480,train,b26ec49c499c0210cb8d35c412f66ec7,d92dc4571fc5421e
Normal-10058.png,Normal,Normal/images/Normal-10058.png,Normal/masks/Normal-10058.png,True,G09481,train,1a3a593d75bf330d9b9843e903d8821d,d42dd2d35f4ac708
Normal-10059.png,Normal,Normal/images/Normal-10059.png,Normal/masks/Normal-10059.png,True,G09482,train,ae3c675958ac184084bae53b224e53d8,dd2946065e0d6f4b
Normal-1006.png,Normal,Normal/images/Normal-1006.png,Normal/masks/Normal-1006.png,True,G09483,train,36ac577bedc02cbd639ea9a6d74e897a,913cc61b9ccd3d98
Normal-10060.png,Normal,Normal/images/Normal-10060.png,Normal/masks/Normal-10060.png,True,G09484,train,6686c9a0037c3ea0b0ad6241f1eceb84,c02c0e835b5f5e4f
Normal-10061.png,Normal,Normal/images/Normal-10061.png,Normal/masks/Normal-10061.png,True,G09485,train,8f783d2aa61bc46525868304b3b7d1ab,c97cc70b10475b9d
Normal-10062.png,Normal,Normal/images/Normal-10062.png,Normal/masks/Normal-10062.png,True,G09486,train,66f097ecc56f56873fe9d4017916966e,c97d07421b1d4f1d
Normal-10063.png,Normal,Normal/images/Normal-10063.png,Normal/masks/Normal-10063.png,True,G09487,train,58a4d4e045f5a7ae176601d8cd6dbd4f,892a861b1d0f5f5d
Normal-10064.png,Normal,Normal/images/Normal-10064.png,Normal/masks/Normal-10064.png,True,G09488,val,955543b731759ae6fa03dfa79d7aebf8,c93ac65f14491b5d
Normal-10065.png,Normal,Normal/images/Normal-10065.png,Normal/masks/Normal-10065.png,True,G09489,train,27d2bf7b85125197610cf76843e23dab,9978861a586d7ae9
Normal-10066.png,Normal,Normal/images/Normal-10066.png,Normal/masks/Normal-10066.png,True,G09490,val,ca2c05f724a96ebffa338bbbdf4a56e7,916916151b5f793a
Normal-10067.png,Normal,Normal/images/Normal-10067.png,Normal/masks/Normal-10067.png,True,G09491,test,6b4c52a0531f0565a0129323881dc77e,8938d6571e8e5a9c
Normal-10068.png,Normal,Normal/images/Normal-10068.png,Normal/masks/Normal-10068.png,True,G09492,test,8cbd318055ba5d12bd085a9bebae4de8,c938d65e5c4b5949
Normal-10069.png,Normal,Normal/images/Normal-10069.png,Normal/masks/Normal-10069.png,True,G09493,test,c52cc783b452163b2adc3c9bffd652f8,8879471e598b3ba5
Normal-1007.png,Normal,Normal/images/Normal-1007.png,Normal/masks/Normal-1007.png,True,G09494,train,ad3acf599ce6e3784fea662b81e2ecc7,95fcc73a8d2c2c4c
Normal-10070.png,Normal,Normal/images/Normal-10070.png,Normal/masks/Normal-10070.png,True,G09495,train,2c6b2e1ffa5afdf863c49201812f763b,cc2ec3431b4d463f
Normal-10071.png,Normal,Normal/images/Normal-10071.png,Normal/masks/Normal-10071.png,True,G09496,train,d0562c72541bb95f04564b7d5398823b,dd74525b514a1f4a
Normal-10072.png,Normal,Normal/images/Normal-10072.png,Normal/masks/Normal-10072.png,True,G09497,train,57dd9ccc9f921f092c24c859a5d6474e,dd6cc61a086e19ed
Normal-10073.png,Normal,Normal/images/Normal-10073.png,Normal/masks/Normal-10073.png,True,G09498,train,99d238f594d27271d94afc27bf5a3d2a,9978061e1a5d78dd
Normal-10074.png,Normal,Normal/images/Normal-10074.png,Normal/masks/Normal-10074.png,True,G09499,train,5bc39dd06d330d600b4ffb331e754e4f,d42ec35b400f3dce
Normal-10075.png,Normal,Normal/images/Normal-10075.png,Normal/masks/Normal-10075.png,True,G09500,train,0a5914092e91de699363b0dfc1dfd85e,c928861e5b0f1f5b
Normal-10076.png,Normal,Normal/images/Normal-10076.png,Normal/masks/Normal-10076.png,True,G09501,train,162c6ddb7e2ef50d80df15e4dc180dd3,dc3d43431d8e0f3c
Normal-10077.png,Normal,Normal/images/Normal-10077.png,Normal/masks/Normal-10077.png,True,G09502,train,27ddfa163b3ab1e39c40f663397b1235,997b4758455d584b
Normal-10078.png,Normal,Normal/images/Normal-10078.png,Normal/masks/Normal-10078.png,True,G09503,train,c586374c66374148a60f9fae700d19cd,cd2e53435d4c4d4d
Normal-10079.png,Normal,Normal/images/Normal-10079.png,Normal/masks/Normal-10079.png,True,G09504,train,3ad87cdc85a5af9462655880e6f36d56,8979c7861a5ecc1b
Normal-1008.png,Normal,Normal/images/Normal-1008.png,Normal/masks/Normal-1008.png,True,G09505,val,44ee5db579c032f7d91e3636f978cb3f,d1e4e65a2d7c286c
Normal-10080.png,Normal,Normal/images/Normal-10080.png,Normal/masks/Normal-10080.png,True,G09506,test,fe3248d4e1fd60a300ae56cf5882d3ab,887b46965a897f98
Normal-10081.png,Normal,Normal/images/Normal-10081.png,Normal/masks/Normal-10081.png,True,G09507,train,8e26d55ea1be715e083bc6b942b299f5,d52c07185cfa79e8
Normal-10082.png,Normal,Normal/images/Normal-10082.png,Normal/masks/Normal-10082.png,True,G09508,train,65cc8242bd9b8482b5ad74c4dbaa8130,9978060b1f6e78b9
Normal-10083.png,Normal,Normal/images/Normal-10083.png,Normal/masks/Normal-10083.png,True,G09509,train,a122789fc39c566b149238b9e11e4cc5,996f86184d5a495f
Normal-10084.png,Normal,Normal/images/Normal-10084.png,Normal/masks/Normal-10084.png,True,G09510,val,a49300d0bf9204c225168ef74595e91b,814d065e4d4f4b5f
Normal-10085.png,Normal,Normal/images/Normal-10085.png,Normal/masks/Normal-10085.png,True,G09511,train,dec15295c3241d233e6d663b4e9ebc54,d42fc34b4d4e4c6c
Normal-10086.png,Normal,Normal/images/Normal-10086.png,Normal/masks/Normal-10086.png,True,G09512,val,c21ef126eb0717e97e40b1e2e3dde054,dc23530f430f4e4f
Normal-10087.png,Normal,Normal/images/Normal-10087.png,Normal/masks/Normal-10087.png,True,G09513,test,117de91f0ccb87b1b24564e161b87197,887b05165e557a7a
Normal-10088.png,Normal,Normal/images/Normal-10088.png,Normal/masks/Normal-10088.png,True,G09514,test,6fa5f511bf1a2019dde25ecbb2e8909e,c86c071a19cd6dfc
Normal-10089.png,Normal,Normal/images/Normal-10089.png,Normal/masks/Normal-10089.png,True,G09515,train,b4d6591f2ccd56185b474609f17b30c3,813f86123a1d6b3f
Normal-1009.png,Normal,Normal/images/Normal-1009.png,Normal/masks/Normal-1009.png,True,G09516,val,86385606303bf783a47c259839323076,8129c67af8693979
Normal-10090.png,Normal,Normal/images/Normal-10090.png,Normal/masks/Normal-10090.png,True,G09517,train,11a362b39f16732500bca20a93691018,807b171e18291fbf
Normal-10091.png,Normal,Normal/images/Normal-10091.png,Normal/masks/Normal-10091.png,True,G09518,train,d6c3fa7b86dbd031f7c5eda353f5c546,d54ac30f323ab333
Normal-10092.png,Normal,Normal/images/Normal-10092.png,Normal/masks/Normal-10092.png,True,G09519,train,1caff7439f333990159842da660158ec,885f425f1a1d3d39
Normal-10093.png,Normal,Normal/images/Normal-10093.png,Normal/masks/Normal-10093.png,True,G09520,train,6f791ae31423efe150a52a59a1acb0a5,857c87c6125f4b1e
Normal-10094.png,Normal,Normal/images/Normal-10094.png,Normal/masks/Normal-10094.png,True,G09521,train,d63f1fa3769f522bac2430004c61bfdd,9917161e5a3a7a3a
Normal-10095.png,Normal,Normal/images/Normal-10095.png,Normal/masks/Normal-10095.png,True,G09522,test,83a4c7062e3bb6e1fc4bc2a4a32ce821,997f045a19487af9
Normal-10096.png,Normal,Normal/images/Normal-10096.png,Normal/masks/Normal-10096.png,True,G09523,train,91d9627d91337035e7c904abe8debf00,dd2e524b1d0f0d1d
Normal-10097.png,Normal,Normal/images/Normal-10097.png,Normal/masks/Normal-10097.png,True,G09524,train,5f8b03d6cfdb0e94c37414bf91900a13,dd6ed0134b4d3c54
Normal-10098.png,Normal,Normal/images/Normal-10098.png,Normal/masks/Normal-10098.png,True,G09525,train,e24d9c2f619264ecd7d498db21546aeb,dd2cd35bc14b1c46
Normal-10099.png,Normal,Normal/images/Normal-10099.png,Normal/masks/Normal-10099.png,True,G09526,train,df52989d7167e34cc04dab153e09c803,8968871b0e5f3dc5
Normal-101.png,Normal,Normal/images/Normal-101.png,Normal/masks/Normal-101.png,True,G09527,train,dc04e4391646a8472eadaf3ebefc8ed8,9139c71a9acb3999
Normal-1010.png,Normal,Normal/images/Normal-1010.png,Normal/masks/Normal-1010.png,True,G09528,train,338f9442f0d89fb75bf4dbd8c85ac522,806bc71aedc569d1
Normal-10100.png,Normal,Normal/images/Normal-10100.png,Normal/masks/Normal-10100.png,True,G09529,train,04f785f805c24a2120e7f2c815fdaf9d,d43fd3431f10cb0b
Normal-10101.png,Normal,Normal/images/Normal-10101.png,Normal/masks/Normal-10101.png,True,G09530,val,e4e9e0559649c381a3bd60b089913c35,9979861a115f39c7
Normal-10102.png,Normal,Normal/images/Normal-10102.png,Normal/masks/Normal-10102.png,True,G09531,val,f6989a9f38c26d9880439fe74eea2528,ddacc21f461b594c
Normal-10103.png,Normal,Normal/images/Normal-10103.png,Normal/masks/Normal-10103.png,True,G09532,test,f1288880b5ba3e9c608b2778d2cd4d9b,9978061e5fe878e1
Normal-10104.png,Normal,Normal/images/Normal-10104.png,Normal/masks/Normal-10104.png,True,G09533,train,d94ecb896c2c62bd25a17c63e83bd747,887907165a5d7d78
Normal-10105.png,Normal,Normal/images/Normal-10105.png,Normal/masks/Normal-10105.png,True,G09534,test,f3a57c53815d1df6d312d5190dbad694,cc3d43531f421f1d
Normal-10106.png,Normal,Normal/images/Normal-10106.png,Normal/masks/Normal-10106.png,True,G09535,train,bcd4d0cd27f692ef0a7e44bd4847df5e,d82984865e4f575b
Normal-10107.png,Normal,Normal/images/Normal-10107.png,Normal/masks/Normal-10107.png,True,G09536,train,91483663b09e1b5133e4263e6f65e105,ccb5c11ec33d706c
Normal-10108.png,Normal,Normal/images/Normal-10108.png,Normal/masks/Normal-10108.png,True,G09537,train,517eb2796c3451580214eeb47dfdfd72,887987961c1d4f5e
Normal-10109.png,Normal,Normal/images/Normal-10109.png,Normal/masks/Normal-10109.png,True,G09538,train,8c46d9028be559d45e2bb11b55b788c4,d93904161e1f595f
Normal-1011.png,Normal,Normal/images/Normal-1011.png,Normal/masks/Normal-1011.png,True,G09539,train,3eaef0ebbbbf5fbd7a02629f999118f0,d029c23ffc483d99
Normal-10110.png,Normal,Normal/images/Normal-10110.png,Normal/masks/Normal-10110.png,True,G09540,val,765d07a25a72fa1bcf08676d49069963,dd4a430f480e4faf
Normal-10111.png,Normal,Normal/images/Normal-10111.png,Normal/masks/Normal-10111.png,True,G09541,test,daded22bc9b1587554dc79de3d190a9a,817907961e5edd58
Normal-10112.png,Normal,Normal/images/Normal-10112.png,Normal/masks/Normal-10112.png,True,G09542,train,9a7ed4911458f4f2bea5da1d20020352,990e161a497a6efa
Normal-10113.png,Normal,Normal/images/Normal-10113.png,Normal/masks/Normal-10113.png,True,G09543,train,38f9d2591c01747cae173dd1a12d42d3,8d2c421bd74f195d
Normal-10114.png,Normal,Normal/images/Normal-10114.png,Normal/masks/Normal-10114.png,True,G09544,test,52769e86f0a0614f4f362c973a8f5c80,c13ad2cb9a4c7e1a
Normal-10115.png,Normal,Normal/images/Normal-10115.png,Normal/masks/Normal-10115.png,True,G09545,train,17506c47ca0b8f0c734fc0fa8d958c6f,9ce78349971f070d
Normal-10116.png,Normal,Normal/images/Normal-10116.png,Normal/masks/Normal-10116.png,True,G09546,train,09ce464d0716c5f197b45574e88cb9a5,9929861a1a5f5b79
Normal-10117.png,Normal,Normal/images/Normal-10117.png,Normal/masks/Normal-10117.png,True,G09547,train,113aeb3ed5610a1e62bc2d89003de6a0,992d06121bd77e39
Normal-10118.png,Normal,Normal/images/Normal-10118.png,Normal/masks/Normal-10118.png,True,G09548,train,ddf04685390cc9ad593cd513490ccb98,dd68171a0f1a5d78
Normal-10119.png,Normal,Normal/images/Normal-10119.png,Normal/masks/Normal-10119.png,True,G09549,train,94b4504cb619baf964c29707dd2136d7,997b061a3969e8b9
Normal-1012.png,Normal,Normal/images/Normal-1012.png,Normal/masks/Normal-1012.png,True,G09550,train,f30a7a772cb2e5db906f964c381822af,d02bc31bbce83cb1
Normal-10120.png,Normal,Normal/images/Normal-10120.png,Normal/masks/Normal-10120.png,True,G09551,test,81da9b48945c5ee11b3052bce4cf9f14,dd2ed2514b454b1d
Normal-10121.png,Normal,Normal/images/Normal-10121.png,Normal/masks/Normal-10121.png,True,G09552,val,ef589ed810b61ca5edf380e876eb2b26,d82a565f124f3ce8
Normal-10122.png,Normal,Normal/images/Normal-10122.png,Normal/masks/Normal-10122.png,True,G09553,val,2ba58c95488d64588488b3f713bde9bb,c809121e5c5c5bff
Normal-10123.png,Normal,Normal/images/Normal-10123.png,Normal/masks/Normal-10123.png,True,G09554,train,df02cb5b18967c1c5fee9f7e36926d9e,dd3cd30b4d0e38e4
Normal-10124.png,Normal,Normal/images/Normal-10124.png,Normal/masks/Normal-10124.png,True,G09555,train,671a8f036fe2a20471853f320ef293d8,887b05941e1a7fba
Normal-10125.png,Normal,Normal/images/Normal-10125.png,Normal/masks/Normal-10125.png,True,G09556,train,8939f266e371e0f5de7c880643f95388,d92dd45215c61b9d
Normal-10126.png,Normal,Normal/images/Normal-10126.png,Normal/masks/Normal-10126.png,True,G09557,train,485aa3c130fb0f11c7ef697a3ef8bafa,993dc61e954e1ac9
Normal-10127.png,Normal,Normal/images/Normal-10127.png,Normal/masks/Normal-10127.png,True,G09558,train,5556a5181bc16cca5232613f096e9660,8178861e1b595f7a
Normal-10128.png,Normal,Normal/images/Normal-10128.png,Normal/masks/Normal-10128.png,True,G09559,train,ee942b776474ece3ad5b275b68db59df,916c461b0f6e386f
Normal-10129.png,Normal,Normal/images/Normal-10129.png,Normal/masks/Normal-10129.png,True,G09560,train,ae9323badad5e7b83f98f4ba356c3dde,d97f461a4a1b390b
Normal-1013.png,Normal,Normal/images/Normal-1013.png,Normal/masks/Normal-1013.png,True,G09561,train,130a09306d3211d2850076ee8788bae9,913bcf1a9a491d4d
Normal-10130.png,Normal,Normal/images/Normal-10130.png,Normal/masks/Normal-10130.png,True,G09562,train,9734f431fac86c9d1dfa7d3deb343cd7,d92ec60b1ceda8ac
Normal-10131.png,Normal,Normal/images/Normal-10131.png,Normal/masks/Normal-10131.png,True,G09563,val,a16446f15b916a7b0f697f824eb9e0c8,897dc61a955f1a45
Normal-10132.png,Normal,Normal/images/Normal-10132.png,Normal/masks/Normal-10132.png,True,G09564,val,4bc70621d1f2ebeba1f8b8eb3685509c,d11b9adb39aad484
Normal-10133.png,Normal,Normal/images/Normal-10133.png,Normal/masks/Normal-10133.png,True,G09565,train,c1eee25c385e16d2604436120e6261a9,dc3dc31f440f384b
Normal-10134.png,Normal,Normal/images/Normal-10134.png,Normal/masks/Normal-10134.png,True,G09566,train,62a27c52eadc80a3685febe993227c6d,c97f461a575a1949
Normal-10135.png,Normal,Normal/images/Normal-10135.png,Normal/masks/Normal-10135.png,True,G09567,train,205a7b5614381452b5d81992d2a05803,dd28920f461f4b5d
Normal-10136.png,Normal,Normal/images/Normal-10136.png,Normal/masks/Normal-10136.png,True,G09568,train,ba09895b06e5468f52d8b0f9ff44ff6f,dc2d42530b4d6e3d
Normal-10137.png,Normal,Normal/images/Normal-10137.png,Normal/masks/Normal-10137.png,True,G09569,val,f5f85ba4dd898d155ff0f6a7fb2ebf02,d13956865e1acd4b
Normal-10138.png,Normal,Normal/images/Normal-10138.png,Normal/masks/Normal-10138.png,True,G09570,test,16501fe6182b9a7f0b6c26221c43efd7,816856195e6f392f
Normal-10139.png,Normal,Normal/images/Normal-10139.png,Normal/masks/Normal-10139.png,True,G09571,train,f32ca4d46d94724227424fa125c87a45,d13946c61b11cd5f
Normal-1014.png,Normal,Normal/images/Normal-1014.png,Normal/masks/Normal-1014.png,True,G09572,test,1bfe46990a74074935ebbbbc2c7266f5,9129c75abaca39b1
Normal-10140.png,Normal,Normal/images/Normal-10140.png,Normal/masks/Normal-10140.png,True,G09573,train,64a5b5351e55d6fff92ae18eef1cac83,c97d061e1f493b89
Normal-10141.png,Normal,Normal/images/Normal-10141.png,Normal/masks/Normal-10141.png,True,G09574,test,df03598d21487b65a110d23cd8793b72,dd2cd2531f441a1f
Normal-10142.png,Normal,Normal/images/Normal-10142.png,Normal/masks/Normal-10142.png,True,G09575,train,e09ff2eecc6f24191385564e335019d1,dd2ec21b044a1f6f
Normal-10143.png,Normal,Normal/images/Normal-10143.png,Normal/masks/Normal-10143.png,True,G09576,test,2297f991a7c9de5b2b95b3567c71fbf4,f925165a757a6162
Normal-10144.png,Normal,Normal/images/Normal-10144.png,Normal/masks/Normal-10144.png,True,G09577,train,2c2d24b92fa626d99e94dffc53c24114,983d061a095b79fb
Normal-10145.png,Normal,Normal/images/Normal-10145.png,Normal/masks/Normal-10145.png,True,G09578,train,47c96a95ea289cd5d8dca48dcea4aa82,dd2ad34b0d6c6c1c
Normal-10146.png,Normal,Normal/images/Normal-10146.png,Normal/masks/Normal-10146.png,True,G09579,train,4cbef5f612e26bb61a49f6900d7a3fbf,d959861a1969bae5
Normal-10147.png,Normal,Normal/images/Normal-10147.png,Normal/masks/Normal-10147.png,True,G09580,train,0b8321043d1dd22b274a48bb7ad1587f,997807161f0f1e3b
Normal-10148.png,Normal,Normal/images/Normal-10148.png,Normal/masks/Normal-10148.png,True,G09581,train,23e69b264afab78527ac23859f93cda9,dd2ed2171b411f19
Normal-10149.png,Normal,Normal/images/Normal-10149.png,Normal/masks/Normal-10149.png,True,G09582,train,9e68859230b20c9c390b1cdabeaa0d98,887d061e59693ced
Normal-1015.png,Normal,Normal/images/Normal-1015.png,Normal/masks/Normal-1015.png,True,G09583,train,9c44c5aa8718a634c9fe77a5ff9f626f,d47cc75b8d8d28a4
Normal-10150.png,Normal,Normal/images/Normal-10150.png,Normal/masks/Normal-10150.png,True,G09584,train,4a50e7d4427eac54aa43b100068382b6,dd2c421b1d4f38f8
Normal-10151.png,Normal,Normal/images/Normal-10151.png,Normal/masks/Normal-10151.png,True,G09585,train,dfe0144db2b99c9e3b5c9d70aa9770a2,d9794611934f1b1b
Normal-10152.png,Normal,Normal/images/Normal-10152.png,Normal/masks/Normal-10152.png,True,G09586,train,4c8418cd31dfce036ed67c0790555824,817986161b5fde1a
Normal-10153.png,Normal,Normal/images/Normal-10153.png,Normal/masks/Normal-10153.png,True,G09587,val,26a87fc24e957f6e220f5ffb137f3a7c,9968871a1dde7a19
Normal-10154.png,Normal,Normal/images/Normal-10154.png,Normal/masks/Normal-10154.png,True,G09588,val,494e09bf44a9238fbe7f8f2351271392,d92942971e0dfd1a
Normal-10155.png,Normal,Normal/images/Normal-10155.png,Normal/masks/Normal-10155.png,True,G09589,train,ca1e37230d2dd80d9bed56c8c442c35e,dd29c21355c61b3d
Normal-10156.png,Normal,Normal/images/Normal-10156.png,Normal/masks/Normal-10156.png,True,G09590,test,e16374d48bd98708bf66ed8f7b120990,9906161b5b7a7a78
Normal-10157.png,Normal,Normal/images/Normal-10157.png,Normal/masks/Normal-10157.png,True,G09591,train,3489d95362c6b83caf0e1df663f5d048,897d041e5969edb4
Normal-10158.png,Normal,Normal/images/Normal-10158.png,Normal/masks/Normal-10158.png,True,G09592,val,04577c9a66ea8f53fb32eb613b258bb3,8168961a1f5e5a5f
Normal-10159.png,Normal,Normal/images/Normal-10159.png,Normal/masks/Normal-10159.png,True,G09593,test,19aafbd04891c88bf31108ef35326bcc,d83c464b1b4d4e79
Normal-1016.png,Normal,Normal/images/Normal-1016.png,Normal/masks/Normal-1016.png,True,G09594,train,1a1004387666192df2710ec61d845f4f,887f465b434f1a4d
Normal-10160.png,Normal,Normal/images/Normal-10160.png,Normal/masks/Normal-10160.png,True,G09595,train,81efac43a1b64369f1e882b85f64accf,ddaec25f50421b55
Normal-10161.png,Normal,Normal/images/Normal-10161.png,Normal/masks/Normal-10161.png,True,G09596,train,48b3c0b61c51de684e49347758df18f9,dd2fc61bd54710c4
Normal-10162.png,Normal,Normal/images/Normal-10162.png,Normal/masks/Normal-10162.png,True,G09597,train,053d2c7e3314f0539b433e2e278c6a8f,dd6a861f432d1b0d
Normal-10163.png,Normal,Normal/images/Normal-10163.png,Normal/masks/Normal-10163.png,True,G09598,train,c7c017668430d003eba84c50e59025e8,d92fc61a186c79e1
Normal-10164.png,Normal,Normal/images/Normal-10164.png,Normal/masks/Normal-10164.png,True,G09599,train,ca357f2f551fbb42f6da6a0332b76387,c93ec6511b075a1f
Normal-10165.png,Normal,Normal/images/Normal-10165.png,Normal/masks/Normal-10165.png,True,G09600,train,f8816e936bfea7412d6fca42c8d57621,9d2dc61e45185d4f
Normal-10166.png,Normal,Normal/images/Normal-10166.png,Normal/masks/Normal-10166.png,True,G09601,train,8906907e95db7ccaabfce45099498018,913a5e187b3d30b5
Normal-10167.png,Normal,Normal/images/Normal-10167.png,Normal/masks/Normal-10167.png,True,G09602,train,4e2c9c68c9d09a0998b17bedcf1d4a9b,9d3dc61b194e3a98
Normal-10168.png,Normal,Normal/images/Normal-10168.png,Normal/masks/Normal-10168.png,True,G09603,train,fb97b7b2fef32ab7641858c30f34308d,d12f464f186cfd90
Normal-10169.png,Normal,Normal/images/Normal-10169.png,Normal/masks/Normal-10169.png,True,G09604,train,694461c6f1c26c1827c99381ec144ee2,d939971e1c091d5d
Normal-1017.png,Normal,Normal/images/Normal-1017.png,Normal/masks/Normal-1017.png,True,G09605,train,1e4a6171ff3cb7f6be895cdcbd3a5b07,9139c61aed79b8c1
Normal-10170.png,Normal,Normal/images/Normal-10170.png,Normal/masks/Normal-10170.png,True,G09606,train,7f3a3980ae75c946b647a124b0fdce14,b97dc7161c5c185c
Normal-10171.png,Normal,Normal/images/Normal-10171.png,Normal/masks/Normal-10171.png,True,G09607,train,629fad68bd191212e3a3da28190d17c2,d97985961ec53e81
Normal-10172.png,Normal,Normal/images/Normal-10172.png,Normal/masks/Normal-10172.png,True,G09608,val,4965d8075ff3587259e3b58b2cbdce81,985b061e1e193bf9
Normal-10173.png,Normal,Normal/images/Normal-10173.png,Normal/masks/Normal-10173.png,True,G09609,val,fac4b8d599ff685e5456870b5ce4afe3,c83d570b444b3dce
Normal-10174.png,Normal,Normal/images/Normal-10174.png,Normal/masks/Normal-10174.png,True,G09610,test,780836be0b1bb0242e2dcca91e854fe3,d92dc6521f461b1d
Normal-10175.png,Normal,Normal/images/Normal-10175.png,Normal/masks/Normal-10175.png,True,G09611,val,544df87c75fb6edcd370218512c74b8a,ec3ed009d3491d5b
Normal-10176.png,Normal,Normal/images/Normal-10176.png,Normal/masks/Normal-10176.png,True,G09612,test,e421cad03c8778c1f5f2aaf0e02d0e00,d92fc65e0c4e1969
Normal-10177.png,Normal,Normal/images/Normal-10177.png,Normal/masks/Normal-10177.png,True,G09613,test,d0b58e8f2813f7368a5f6dc7411c6db7,d42f43084cee7cec
Normal-10178.png,Normal,Normal/images/Normal-10178.png,Normal/masks/Normal-10178.png,True,G09614,train,9a0ef23f4c6d4887fbc8bddaa0ed56a4,95681e0f5b79e468
Normal-10179.png,Normal,Normal/images/Normal-10179.png,Normal/masks/Normal-10179.png,True,G09615,train,4139ad10eb729f09af2ea36d262f3d0d,f97987971a1c0d1a
Normal-1018.png,Normal,Normal/images/Normal-1018.png,Normal/masks/Normal-1018.png,True,G09616,train,375c641bcbc0f4b59c02c66a92810184,913b863a9859b9cb
Normal-10180.png,Normal,Normal/images/Normal-10180.png,Normal/masks/Normal-10180.png,True,G09617,train,7ceb9b6d6bb5da181f6d80bd90f07784,d92dc61b4d1b394a
Normal-10181.png,Normal,Normal/images/Normal-10181.png,Normal/masks/Normal-10181.png,True,G09618,train,97e712fdb20d7613016c37cfc0af1750,957c8653f90bea09
Normal-10182.png,Normal,Normal/images/Normal-10182.png,Normal/masks/Normal-10182.png,True,G09619,train,714e926a51670862dd478b15a03e97e2,d93b865e1e0c5b19
Normal-10183.png,Normal,Normal/images/Normal-10183.png,Normal/masks/Normal-10183.png,True,G09620,train,52af01434fbb49e17af258b69af7f063,d93b56171b55093a
Normal-10184.png,Normal,Normal/images/Normal-10184.png,Normal/masks/Normal-10184.png,True,G09621,train,cb4498e5b3ec2a3a45dd366c59333125,9c1ad30e0e3a9e3e
Normal-10185.png,Normal,Normal/images/Normal-10185.png,Normal/masks/Normal-10185.png,True,G09622,train,6edfcf0a6e201a883759e8a8c5d25b7f,897905861f155f5e
Normal-10186.png,Normal,Normal/images/Normal-10186.png,Normal/masks/Normal-10186.png,True,G09623,test,fb720935023f1d4a9ee3ef36aaf223b0,d83ac2ca9de4a2b6
Normal-10187.png,Normal,Normal/images/Normal-10187.png,Normal/masks/Normal-10187.png,True,G09624,train,97cbd591509f0d8066367df7fa2dcb55,d5bec359450d3c45
Normal-10188.png,Normal,Normal/images/Normal-10188.png,Normal/masks/Normal-10188.png,True,G09625,train,87f13b2df976e1bf3082b8493cb93911,993985861b17c71f
Normal-10189.png,Normal,Normal/images/Normal-10189.png,Normal/masks/Normal-10189.png,True,G09626,train,2d33ae1550db04a0cfd25d70d7155ec4,997c461a79a43af1
Normal-1019.png,Normal,Normal/images/Normal-1019.png,Normal/masks/Normal-1019.png,True,G09627,train,7fe93d2520a1f8f4d97b5b380b71ae26,d12dc61e594f3c51
Normal-10190.png,Normal,Normal/images/Normal-10190.png,Normal/masks/Normal-10190.png,True,G09628,train,1a3b4ed7ad5cd1f5a0fcdc2b16cd6bfb,dd79c61a444f7845
Normal-10191.png,Normal,Normal/images/Normal-10191.png,Normal/masks/Normal-10191.png,True,G09629,val,7b6a702e072412b7fb2937886c1c5605,d03c034b594b6f3d
Normal-10192.png,Normal,Normal/images/Normal-10192.png,Normal/masks/Normal-10192.png,True,G09630,val,409d891d99594648c90be900c10c8532,dd2c861b19c63af8
Normal-102.png,Normal,Normal/images/Normal-102.png,Normal/masks/Normal-102.png,True,G09631,val,cd820ef8511f0192d382ea03c1a92cf0,917ca73a8f3d280d
Normal-1020.png,Normal,Normal/images/Normal-1020.png,Normal/masks/Normal-1020.png,True,G09632,test,e093b67b34036b7b2217b606552d2405,812bc24addc19bde
Normal-1021.png,Normal,Normal/images/Normal-1021.png,Normal/masks/Normal-1021.png,True,G09633,train,c8191747d9e3578a480a60e63d3bcfff,91edc71aed6910c3
Normal-1022.png,Normal,Normal/images/Normal-1022.png,Normal/masks/Normal-1022.png,True,G09634,test,b53a32dc7af34a78a4670eeaea8abfd2,992bd25edc591949
Normal-1023.png,Normal,Normal/images/Normal-1023.png,Normal/masks/Normal-1023.png,True,G09635,train,7e83cc84d3faf50d69bfe310ea928e07,d46ec34bc9c578c2
Normal-1024.png,Normal,Normal/images/Normal-1024.png,Normal/masks/Normal-1024.png,True,G09636,train,f9f6ffaf5948fa3c0809599b5a114eb8,9929d61ed8693969
Normal-1025.png,Normal,Normal/images/Normal-1025.png,Normal/masks/Normal-1025.png,True,G09637,train,a677b081dadeafc715d57f0bf8d45bdb,d46fc30bc54d4d46
Normal-1026.png,Normal,Normal/images/Normal-1026.png,Normal/masks/Normal-1026.png,True,G09638,train,17525fbb40905975e210af17a21a2647,9979471e1a474f1c
Normal-1027.png,Normal,Normal/images/Normal-1027.png,Normal/masks/Normal-1027.png,True,G09639,val,69b82d699e9e038e811b3a6cf8f3fb2e,812ec31acd599cd7
Normal-1028.png,Normal,Normal/images/Normal-1028.png,Normal/masks/Normal-1028.png,True,G09640,train,c7f77d4e0340090551255f00516a39f4,d476873b8d2c3c86
Normal-1029.png,Normal,Normal/images/Normal-1029.png,Normal/masks/Normal-1029.png,True,G09641,test,926780d66d1f376797d3a26d6cfa5270,d57f870f8d4d0c84
Normal-103.png,Normal,Normal/images/Normal-103.png,Normal/masks/Normal-103.png,True,G09642,train,94805baa4f5e28a5460d262bb946f831,816ac67a58797d45
Normal-1030.png,Normal,Normal/images/Normal-1030.png,Normal/masks/Normal-1030.png,True,G09643,train,76951d73a20110d8da8354adc661e8e2,903dc71ea8c93dd4
Normal-1031.png,Normal,Normal/images/Normal-1031.png,Normal/masks/Normal-1031.png,True,G09644,train,ab773df95fd11595a3ba0cc55330ae80,816e861acd7d38e5
Normal-1032.png,Normal,Normal/images/Normal-1032.png,Normal/masks/Normal-1032.png,True,G09645,train,aaad59209dce1cc5955bd6bbe72b0598,903fc71a6d493c69
Normal-1033.png,Normal,Normal/images/Normal-1033.png,Normal/masks/Normal-1033.png,True,G09646,train,842028d3248421b8d15e9c798d7f1f0c,d12cc61bbc6b3869
Normal-1034.png,Normal,Normal/images/Normal-1034.png,Normal/masks/Normal-1034.png,True,G09647,train,432f941d4d1bfdec9e7b0a95197fbe90,c1efcf29c54b8243
Normal-1035.png,Normal,Normal/images/Normal-1035.png,Normal/masks/Normal-1035.png,True,G09648,val,441e320a043a5f7f2ca7d49b21965ef4,9129c21bd87999db
Normal-1036.png,Normal,Normal/images/Normal-1036.png,Normal/masks/Normal-1036.png,True,G09649,train,8b073bce64e4fcf837a1e1374af6f277,d46fc72dc56468c2
Normal-1037.png,Normal,Normal/images/Normal-1037.png,Normal/masks/Normal-1037.png,True,G09650,train,60cac0a15262f65fcdcdfa3d89643fcd,913b931adb1e1917
Normal-1038.png,Normal,Normal/images/Normal-1038.png,Normal/masks/Normal-1038.png,True,G09651,val,d0d1d12cd691b006c8f6d4f135bd3d3b,9129c238f979396d
Normal-1039.png,Normal,Normal/images/Normal-1039.png,Normal/masks/Normal-1039.png,True,G09652,test,eaece1ac605b5d132d5f8d77e918b9a7,817e461a5a4d78ed
Normal-104.png,Normal,Normal/images/Normal-104.png,Normal/masks/Normal-104.png,True,G09653,test,c77e6c237c49ed5e4b179b9668d2f6d0,986dc31adb493ccc
Normal-1040.png,Normal,Normal/images/Normal-1040.png,Normal/masks/Normal-1040.png,True,G09654,train,56f2d55aefd525249c0815db4222907a,916cc63acd691d4d
Normal-1041.png,Normal,Normal/images/Normal-1041.png,Normal/masks/Normal-1041.png,True,G09655,train,2e534273f62e2238df0f1c43a5012766,d07cc71e8d6538c6
Normal-1042.png,Normal,Normal/images/Normal-1042.png,Normal/masks/Normal-1042.png,True,G09656,train,3037ec63c07159d02740c2ec9a868fb0,98f5875a8b4f2817
Normal-1043.png,Normal,Normal/images/Normal-1043.png,Normal/masks/Normal-1043.png,True,G09657,train,40ab03581aa2e4f619278cdf868e99aa,95ecce388d4d4cc6
Normal-1044.png,Normal,Normal/images/Normal-1044.png,Normal/masks/Normal-1044.png,True,G09658,train,2d1a0e5a6fc512586e5f1aa304bc228f,9179c71a9a9e3895
Normal-1045.png,Normal,Normal/images/Normal-1045.png,Normal/masks/Normal-1045.png,True,G09659,train,5f97df44c168d840ad368d2d70bd2156,913bd65a585f1d90
Normal-1046.png,Normal,Normal/images/Normal-1046.png,Normal/masks/Normal-1046.png,True,G09660,train,21cfe77699654b7f8b332ba98245e3a6,9129c33af8697965
Normal-1047.png,Normal,Normal/images/Normal-1047.png,Normal/masks/Normal-1047.png,True,G09661,train,df9099f531a1dbc8a12bf907e7386549,d47cc71b8d8d3c84
Normal-1048.png,Normal,Normal/images/Normal-1048.png,Normal/masks/Normal-1048.png,True,G09662,val,a787fe8e3ecb30675e483246dd97fdf6,c121c23af8693deb
Normal-1049.png,Normal,Normal/images/Normal-1049.png,Normal/masks/Normal-1049.png,True,G09663,test,ce386bbb2d38203de264b3276adad799,99fd873a8b2e380c
Normal-105.png,Normal,Normal/images/Normal-105.png,Normal/masks/Normal-105.png,True,G09664,train,98efa59ae27113df893738df5d904adf,859fce1ae10bb1c6
Normal-1050.png,Normal,Normal/images/Normal-1050.png,Normal/masks/Normal-1050.png,True,G09665,train,bef2014ccd880ec78557bec6da43be75,917d8e1a0d4b794d
Normal-1051.png,Normal,Normal/images/Normal-1051.png,Normal/masks/Normal-1051.png,True,G09666,train,64d6afcb04b203c80c4317ce025ce969,9839c61ada4a3f78
Normal-1052.png,Normal,Normal/images/Normal-1052.png,Normal/masks/Normal-1052.png,True,G09667,train,68ab4239612ec217c17ebf922df13e1f,9dbfc20b9a4b3c30
Normal-1053.png,Normal,Normal/images/Normal-1053.png,Normal/masks/Normal-1053.png,True,G09668,train,726fa31452cd5b399ad5d22dfeff2e7d,d467c74b854ce4e4
Normal-1054.png,Normal,Normal/images/Normal-1054.png,Normal/masks/Normal-1054.png,True,G09669,train,71f4958e31804f860f2d01037dfd0c00,d03cc73b9d4e3990
Normal-1055.png,Normal,Normal/images/Normal-1055.png,Normal/masks/Normal-1055.png,True,G09670,val,1af043086af7289c7f850742c56fc9f8,946e8f1acd0d8c4f
Normal-1056.png,Normal,Normal/images/Normal-1056.png,Normal/masks/Normal-1056.png,True,G09671,val,b275ab2479fe26b8b805d29c32e73dba,9969c65a595d5915
Normal-1057.png,Normal,Normal/images/Normal-1057.png,Normal/masks/Normal-1057.png,True,G09672,test,409dd5a80ea9286c75109fd85a96d4a2,9839c65ada591979
Normal-1058.png,Normal,Normal/images/Normal-1058.png,Normal/masks/Normal-1058.png,True,G09673,train,23d77f094140b9a12d2749b9c952aaeb,917c86388d6939cf
Normal-1059.png,Normal,Normal/images/Normal-1059.png,Normal/masks/Normal-1059.png,True,G09674,test,f1e6bf9b9cfd92ec7479655e22c40269,917dc71e8d4919cc
Normal-106.png,Normal,Normal/images/Normal-106.png,Normal/masks/Normal-106.png,True,G09675,train,630a054de13da2729070b5486526c22d,d02dc71b89cd3ac6
Normal-1060.png,Normal,Normal/images/Normal-1060.png,Normal/masks/Normal-1060.png,True,G09676,train,00043fc8a5db773ce8f4046bb7d816b2,947ec31bbc8c3cb0
Normal-1061.png,Normal,Normal/images/Normal-1061.png,Normal/masks/Normal-1061.png,True,G09677,train,2c84f1daae684f8ec88bc3f1546b0989,91b5c63acb6d3944
Normal-1062.png,Normal,Normal/images/Normal-1062.png,Normal/masks/Normal-1062.png,True,G09678,train,03c74df0915315de4c240a4c3a504dde,d42cc73dcc0c1ccf
Normal-1063.png,Normal,Normal/images/Normal-1063.png,Normal/masks/Normal-1063.png,True,G09679,train,47dc04b7665fc5fc76758f783e829b06,907dc61a98dd1cd6
Normal-1064.png,Normal,Normal/images/Normal-1064.png,Normal/masks/Normal-1064.png,True,G09680,train,abb501c8b646c54d5725c6fe07ce9b00,992f42196c3fb0e3
Normal-1065.png,Normal,Normal/images/Normal-1065.png,Normal/masks/Normal-1065.png,True,G09681,train,163fd7bfef0b73eed0bdb44a3f42fc99,817fc65e18c94f15
Normal-1066.png,Normal,Normal/images/Normal-1066.png,Normal/masks/Normal-1066.png,True,G09682,train,b9ec660af6530fa4fe0824738d7bfedf,d56c871ecf2d28c4
Normal-1067.png,Normal,Normal/images/Normal-1067.png,Normal/masks/Normal-1067.png,True,G09683,train,bd71259aabaa29ec13fbecbc0d56f661,dc2cc31bc94e7c4c
Normal-1068.png,Normal,Normal/images/Normal-1068.png,Normal/masks/Normal-1068.png,True,G09684,train,2bdd796c9c3507f41593990772d12ea8,d02eeb4b9e481dcc
Normal-1069.png,Normal,Normal/images/Normal-1069.png,Normal/masks/Normal-1069.png,True,G09685,train,51af310b20b59c980ba0e2307e8b7f24,c12bc20afc693d3b
Normal-107.png,Normal,Normal/images/Normal-107.png,Normal/masks/Normal-107.png,True,G09686,train,391e5481307e5cf55bb08efcf93a95d2,c5eec74bc96c68c0
Normal-1070.png,Normal,Normal/images/Normal-1070.png,Normal/masks/Normal-1070.png,True,G09687,val,690e18b613bf22fe44d955a04eee6f8f,d03dc6ce9cc823b6
Normal-1071.png,Normal,Normal/images/Normal-1071.png,Normal/masks/Normal-1071.png,True,G09688,val,03d3f7aa36d49d5b8361078999911acb,c03dc338ec6b3cc9
Normal-1072.png,Normal,Normal/images/Normal-1072.png,Normal/masks/Normal-1072.png,True,G09689,val,8d8bbaa9e3c6e0f912f92d4821e6dba5,8139e63af8793869
Normal-1073.png,Normal,Normal/images/Normal-1073.png,Normal/masks/Normal-1073.png,True,G09690,train,c5f35934ce072c2dd179c80d9b714119,907bc65eda591915
Normal-1074.png,Normal,Normal/images/Normal-1074.png,Normal/masks/Normal-1074.png,True,G09691,test,1ec49fee0725f40c1ba92da8420737fb,d56f872b0d0d4cc5
Normal-1075.png,Normal,Normal/images/Normal-1075.png,Normal/masks/Normal-1075.png,True,G09692,test,782a5cbe44b0dda221fed03b9860bb34,913bc21ad85939db
Normal-1076.png,Normal,Normal/images/Normal-1076.png,Normal/masks/Normal-1076.png,True,G09693,test,4b40fa4a64eee19de6e9ac8547673f8b,9135c63ab0bd3995
Normal-1077.png,Normal,Normal/images/Normal-1077.png,Normal/masks/Normal-1077.png,True,G09694,train,4faece82692e1df5e421810ace9df98c,906dc61ee9493dcc
Normal-1078.png,Normal,Normal/images/Normal-1078.png,Normal/masks/Normal-1078.png,True,G09695,train,8e4f635d6b45c103d92cea0002062936,9174c63bad3d388c
Normal-1079.png,Normal,Normal/images/Normal-1079.png,Normal/masks/Normal-1079.png,True,G09696,train,56c77f0a7adda87b6b133fb7ebd4e3ad,d475871f8b0d68c6
Normal-108.png,Normal,Normal/images/Normal-108.png,Normal/masks/Normal-108.png,True,G09697,train,6f72856e859e34883cf9e90dae6d254b,d0fe872cad8e2986
Normal-1080.png,Normal,Normal/images/Normal-1080.png,Normal/masks/Normal-1080.png,True,G09698,train,edf1e61ba6db13c145dd53a2f0953ccb,d03cc71b9d493ccc
Normal-1081.png,Normal,Normal/images/Normal-1081.png,Normal/masks/Normal-1081.png,True,G09699,train,1b002f22b48e8895f49b558bb96e28c9,c10fc61adce93cd2
Normal-1082.png,Normal,Normal/images/Normal-1082.png,Normal/masks/Normal-1082.png,True,G09700,train,7845ccdf74d156d5d4ec430b07a241c1,d47ec71bbd0d2884
Normal-1083.png,Normal,Normal/images/Normal-1083.png,Normal/masks/Normal-1083.png,True,G09701,train,d0883bc2580568212ef4286f121e65a1,913fc61ab8c93995
Normal-1084.png,Normal,Normal/images/Normal-1084.png,Normal/masks/Normal-1084.png,True,G09702,train,364e3ab6a1ee4aec58f4fc2d5b577dfa,912ac23ad97939cb
Normal-1085.png,Normal,Normal/images/Normal-1085.png,Normal/masks/Normal-1085.png,True,G09703,train,69a27dbcd91cab3bd30a01d3abadbc0e,9139c63ab9491cde
Normal-1086.png,Normal,Normal/images/Normal-1086.png,Normal/masks/Normal-1086.png,True,G09704,train,92fef3813b2491d5e4ca3704cbdb2fb0,912bca1ad8499fda
Normal-1087.png,Normal,Normal/images/Normal-1087.png,Normal/masks/Normal-1087.png,True,G09705,train,30ab5355374b3bba7c5a00d3d6564a12,d02dc60fcc6d3cc6
Normal-1088.png,Normal,Normal/images/Normal-1088.png,Normal/masks/Normal-1088.png,True,G09706,train,10479332a5c8637ea3bf887fff54ba06,c13ee31b6de069c4
Normal-1089.png,Normal,Normal/images/Normal-1089.png,Normal/masks/Normal-1089.png,True,G09707,train,2e9e3a9c80eb087e3e1eabf507164f54,9178ca3a597978d1
Normal-109.png,Normal,Normal/images/Normal-109.png,Normal/masks/Normal-109.png,True,G09708,train,ed0e168a631f66210a2a426b6eb1ad8a,d43f870b894b19e5
Normal-1090.png,Normal,Normal/images/Normal-1090.png,Normal/masks/Normal-1090.png,True,G09709,val,7fb637482d949fa8258220ed27110c01,910bc21ad8693fe7
Normal-1091.png,Normal,Normal/images/Normal-1091.png,Normal/masks/Normal-1091.png,True,G09710,val,6e9a7df5856c314e329c6d29a10bbeb0,d474c24bdfc968a4
Normal-1092.png,Normal,Normal/images/Normal-1092.png,Normal/masks/Normal-1092.png,True,G09711,val,f12d36a22ab8b1674749a8a7d24a999e,d4acc70fcf2d0c68
Normal-1093.png,Normal,Normal/images/Normal-1093.png,Normal/masks/Normal-1093.png,True,G09712,test,bd394a2d0629fa7e2caad79b7865396a,d82d421f5a4d1e7c
Normal-1094.png,Normal,Normal/images/Normal-1094.png,Normal/masks/Normal-1094.png,True,G09713,test,90d427af30e888c16a90ad7320a07416,816fce5e5d491d44
Normal-1095.png,Normal,Normal/images/Normal-1095.png,Normal/masks/Normal-1095.png,True,G09714,test,d015a5f0724a706c86a0a7cb09e32e98,d57cc31bcbc168d0
Normal-1096.png,Normal,Normal/images/Normal-1096.png,Normal/masks/Normal-1096.png,True,G09715,train,f22be774f70b041cb844dfdd5d3629a2,d868c25ed75f180d
Normal-1097.png,Normal,Normal/images/Normal-1097.png,Normal/masks/Normal-1097.png,True,G09716,train,8198d1d9737cbdfa5c0dee681276163c,913bc63a9959199b
Normal-1098.png,Normal,Normal/images/Normal-1098.png,Normal/masks/Normal-1098.png,True,G09717,train,9b66adc1c3be9153301dd3869a5836f9,903bd61c9429b5e7
Normal-1099.png,Normal,Normal/images/Normal-1099.png,Normal/masks/Normal-1099.png,True,G09718,train,a2501a388fe97b54d447d86ee1456d23,993bd61e104b396b
Normal-11.png,Normal,Normal/images/Normal-11.png,Normal/masks/Normal-11.png,True,G09719,train,b01370644f490c151ca05b25adf7dd2f,9968861a0b5b5b5f
Normal-110.png,Normal,Normal/images/Normal-110.png,Normal/masks/Normal-110.png,True,G09720,train,94eecdd7a35a22dc39575943c11d9bd0,91eece5acd19c11c
Normal-1100.png,Normal,Normal/images/Normal-1100.png,Normal/masks/Normal-1100.png,True,G09721,train,5265d5ba1f40f80434a616b62a5625e1,9d6d871aad2d3c24
Normal-1101.png,Normal,Normal/images/Normal-1101.png,Normal/masks/Normal-1101.png,True,G09722,train,d89fbc69fb9e1422a85add363e4b75f1,d47cc31f8d0c1cce
Normal-1102.png,Normal,Normal/images/Normal-1102.png,Normal/masks/Normal-1102.png,True,G09723,train,8bed3ef6ea1c2dd52625eb5bd303adbd,d464c34fcd6d38e0
Normal-1103.png,Normal,Normal/images/Normal-1103.png,Normal/masks/Normal-1103.png,True,G09724,train,e6c51bac70882b98034619d85d50ccec,996f8f3a0d284d4d
Normal-1104.png,Normal,Normal/images/Normal-1104.png,Normal/masks/Normal-1104.png,True,G09725,train,a7082c6a09ff36ef665bd592677663dc,907fa71ab93138b4
Normal-1105.png,Normal,Normal/images/Normal-1105.png,Normal/masks/Normal-1105.png,True,G09726,train,7d505270b3c24ab75a151bbb973d61c0,9131c63999ed3999
Normal-1106.png,Normal,Normal/images/Normal-1106.png,Normal/masks/Normal-1106.png,True,G09727,val,4bd720a1a642b3cd81ca2a64b7b50f4e,902cc75aed693c6c
Normal-1107.png,Normal,Normal/images/Normal-1107.png,Normal/masks/Normal-1107.png,True,G09728,train,c422f4b40da6f75c135e9eefaece28f8,d43de63eed2c3840
Normal-1108.png,Normal,Normal/images/Normal-1108.png,Normal/masks/Normal-1108.png,True,G09729,train,72bd307d5dbeddc54338c77eeb6778ac,917d873aa919394d
Normal-1109.png,Normal,Normal/images/Normal-1109.png,Normal/masks/Normal-1109.png,True,G09730,train,ac6dc05577d2d74c2dc3989e083e9c6e,81bfc21aecc93cc9
Normal-111.png,Normal,Normal/images/Normal-111.png,Normal/masks/Normal-111.png,True,G09731,val,a4efb144a6c4d3a4d829e7de3146200a,917ece7889c71386
Normal-1110.png,Normal,Normal/images/Normal-1110.png,Normal/masks/Normal-1110.png,True,G09732,val,3ee26671965c26a82c8faf43e0cf63f9,912bc21b9c693df2
Normal-1111.png,Normal,Normal/images/Normal-1111.png,Normal/masks/Normal-1111.png,True,G09733,test,1005e020fd384bbfef4852cc5f418357,913ce21ade491cfc
Normal-1112.png,Normal,Normal/images/Normal-1112.png,Normal/masks/Normal-1112.png,True,G09734,test,1e2cc545172ef00752270db33b5ec3d7,dc2c425d5b6e386c
Normal-1113.png,Normal,Normal/images/Normal-1113.png,Normal/masks/Normal-1113.png,True,G09735,test,12cfebbea25a851572010507844bd75d,d07c875abded2884
Normal-1114.png,Normal,Normal/images/Normal-1114.png,Normal/masks/Normal-1114.png,True,G09736,val,20c7cd2ad77f0c0b9374880686dd234b,912dc61af8693dc9
Normal-1115.png,Normal,Normal/images/Normal-1115.png,Normal/masks/Normal-1115.png,True,G09737,train,49a07446bc46cf3567a165c482b44d1c,d03ec30ddc6c3cc6
Normal-1116.png,Normal,Normal/images/Normal-1116.png,Normal/masks/Normal-1116.png,True,G09738,train,9d264abb07ceccaaec8f533c109ebacc,d47e873b8d2d2886
Normal-1117.png,Normal,Normal/images/Normal-1117.png,Normal/masks/Normal-1117.png,True,G09739,test,9c30188f13ab6454e48abd9c233362d4,d52fa70b014d6cc7
Normal-1118.png,Normal,Normal/images/Normal-1118.png,Normal/masks/Normal-1118.png,True,G09740,train,9979ee99da391eb85c94446c3b44a199,906bc65af96d384c
Normal-1119.png,Normal,Normal/images/Normal-1119.png,Normal/masks/Normal-1119.png,True,G09741,train,99821e669572b22d3d9fb99a4c34facc,99b9c61eda5a184b
Normal-112.png,Normal,Normal/images/Normal-112.png,Normal/masks/Normal-112.png,True,G09742,train,8360991684008da410745b0d1ac119d7,9139c71b99cd7c90
Normal-1120.png,Normal,Normal/images/Normal-1120.png,Normal/masks/Normal-1120.png,True,G09743,train,414aa0f80962e3a1dc86d7cdbb615564,9139c638b9f93994
Normal-1121.png,Normal,Normal/images/Normal-1121.png,Normal/masks/Normal-1121.png,True,G09744,train,7f649a551f64906e2ffc0195e64a4760,9429c21bf97d19d1
Normal-1122.png,Normal,Normal/images/Normal-1122.png,Normal/masks/Normal-1122.png,True,G09745,train,a5c0a2e3ec26f3435eaf59215b12a3ef,912dc71ef84939e1
Normal-1123.png,Normal,Normal/images/Normal-1123.png,Normal/masks/Normal-1123.png,True,G09746,train,87345b45211003ea4b272309e930507b,912bda5a58591cd7
Normal-1124.png,Normal,Normal/images/Normal-1124.png,Normal/masks/Normal-1124.png,True,G09747,train,92623719ea24551c1ccd954bc57500be,816eca58cd499ecb
Normal-1125.png,Normal,Normal/images/Normal-1125.png,Normal/masks/Normal-1125.png,True,G09748,train,706b2f7a942c5e6257d241ed3f96623f,c12ae64afcc85dd4
Normal-1126.png,Normal,Normal/images/Normal-1126.png,Normal/masks/Normal-1126.png,True,G09749,train,0d946c53e0fb8f0f345d6a4bad1c465e,91adc63acb69386c
Normal-1127.png,Normal,Normal/images/Normal-1127.png,Normal/masks/Normal-1127.png,True,G09750,train,dc18be8daebac59825498c85d6590a85,9179863ab82d39ad
Normal-1128.png,Normal,Normal/images/Normal-1128.png,Normal/masks/Normal-1128.png,True,G09751,train,217e8ca41aa1b99bcbef8400ccbb4c44,9d6e461a4f4c786c
Normal-1129.png,Normal,Normal/images/Normal-1129.png,Normal/masks/Normal-1129.png,True,G09752,val,cc0ce946fbbb3382bc48d79945c89c1f,d07ec71fc9c138b0
Normal-113.png,Normal,Normal/images/Normal-113.png,Normal/masks/Normal-113.png,True,G09753,val,adef4882d9375ca2612d066957a1b218,806ec61cc96d79c7
Normal-1130.png,Normal,Normal/images/Normal-1130.png,Normal/masks/Normal-1130.png,True,G09754,test,145bc7056b979aced1c3035dbf32c6e8,9139ce1e3a994b39
Normal-1131.png,Normal,Normal/images/Normal-1131.png,Normal/masks/Normal-1131.png,True,G09755,val,7baf5dc2601ca41b41984c84dfdb5d99,d47e871bcd2d2886
Normal-1132.png,Normal,Normal/images/Normal-1132.png,Normal/masks/Normal-1132.png,True,G09756,train,c4aec2dfef7a1619ac1aba406c1efa36,917ec63a9b49194d
Normal-1133.png,Normal,Normal/images/Normal-1133.png,Normal/masks/Normal-1133.png,True,G09757,test,ee282606808603e545c13b5580cd581c,d46e870b8d0e58d7
Normal-1134.png,Normal,Normal/images/Normal-1134.png,Normal/masks/Normal-1134.png,True,G09758,train,69bc5dcd309764b1b07ad58d6bc6621b,907e831bdd1c1cd5
Normal-1135.png,Normal,Normal/images/Normal-1135.png,Normal/masks/Normal-1135.png,True,G09759,train,ac55880938acc801910f75227c8a26fb,d42faf2f2c4c0cca
Normal-1136.png,Normal,Normal/images/Normal-1136.png,Normal/masks/Normal-1136.png,True,G09760,train,f3c69523fa2aae7151d0bb20634b7688,943e831b9b1f1cd4
Normal-1137.png,Normal,Normal/images/Normal-1137.png,Normal/masks/Normal-1137.png,True,G09761,train,f4983b3e2bb19b37108e592b2cf3a300,d031c21f997db84d
Normal-1138.png,Normal,Normal/images/Normal-1138.png,Normal/masks/Normal-1138.png,True,G09762,train,3120da4b349ea99539b41bb3c2e06d68,d12ed65b18083dde
Normal-1139.png,Normal,Normal/images/Normal-1139.png,Normal/masks/Normal-1139.png,True,G09763,train,2214b747f3a068f6b28e0d1ee921993b,9c2e025f474f584f
Normal-114.png,Normal,Normal/images/Normal-114.png,Normal/masks/Normal-114.png,True,G09764,train,fb98d4976a430be69b4e2853295fa05c,d47c873a8d4d38ac
Normal-1140.png,Normal,Normal/images/Normal-1140.png,Normal/masks/Normal-1140.png,True,G09765,train,188c7db32070695149f8420994be5818,d467872b852de4c6
Normal-1141.png,Normal,Normal/images/Normal-1141.png,Normal/masks/Normal-1141.png,True,G09766,train,256a21aa853995669241128f7140f003,d12de65b5d491d48
Normal-1142.png,Normal,Normal/images/Normal-1142.png,Normal/masks/Normal-1142.png,True,G09767,train,e0f7ce8efdf434a0ce11fc0d6f5863de,81f1c63ab8c979b1
Normal-1143.png,Normal,Normal/images/Normal-1143.png,Normal/masks/Normal-1143.png,True,G09768,train,a67cbee8bcda599684e29bd50a010ac9,902fc61af8493dd3
Normal-1144.png,Normal,Normal/images/Normal-1144.png,Normal/masks/Normal-1144.png,True,G09769,val,16bd2b08323ea87d942d3442974f5b7e,d476971b893c7c24
Normal-1145.png,Normal,Normal/images/Normal-1145.png,Normal/masks/Normal-1145.png,True,G09770,train,1f0140feda8fff2e78ab59c6daf20c67,d0f4c71ec9c5a0c7
Normal-1146.png,Normal,Normal/images/Normal-1146.png,Normal/masks/Normal-1146.png,True,G09771,train,115ef4ad330e21fd1290c7f33b2e1149,913bd21ad0593dd3
Normal-1147.png,Normal,Normal/images/Normal-1147.png,Normal/masks/Normal-1147.png,True,G09772,val,eef6bec26af8d2fa95cf778e926c6334,d474e67ae96960c4
Normal-1148.png,Normal,Normal/images/Normal-1148.png,Normal/masks/Normal-1148.png,True,G09773,test,3282a22cd33d5803665e8d91c5caee60,896dc65acd4919c7
Normal-1149.png,Normal,Normal/images/Normal-1149.png,Normal/masks/Normal-1149.png,True,G09774,val,86a76de34fedc1e0a2a5b2034ffb919c,916b461b0b2f3d2d
Normal-115.png,Normal,Normal/images/Normal-115.png,Normal/masks/Normal-115.png,True,G09775,test,c0bab67f7e7abac22548d22c36110b55,91b5c63ac87939c3
Normal-1150.png,Normal,Normal/images/Normal-1150.png,Normal/masks/Normal-1150.png,True,G09776,test,d7321d423b183c9e86e4aa72bdf19e44,8129e23b78697979
Normal-1151.png,Normal,Normal/images/Normal-1151.png,Normal/masks/Normal-1151.png,True,G09777,train,27f88522fc2d9518da3fce9ab54df5c4,d029f24b1ef8693c
Normal-1152.png,Normal,Normal/images/Normal-1152.png,Normal/masks/Normal-1152.png,True,G09778,train,7a77de31c8800c176b7df8dfe70b1b74,d42fc74f1d444c9c
Normal-1153.png,Normal,Normal/images/Normal-1153.png,Normal/masks/Normal-1153.png,True,G09779,train,825549f2cafb1981d452bb88ad870fd3,916fcf5e5d4d0818
Normal-1154.png,Normal,Normal/images/Normal-1154.png,Normal/masks/Normal-1154.png,True,G09780,val,3bb524e11f2e390f37b7549f5254d171,d476c70fed242cc4
Normal-1155.png,Normal,Normal/images/Normal-1155.png,Normal/masks/Normal-1155.png,True,G09781,train,866bf410c3e2c271ca9ccb83bb51af87,917ec71badc43894
Normal-1156.png,Normal,Normal/images/Normal-1156.png,Normal/masks/Normal-1156.png,True,G09782,train,f27157960d5e7703092f21b3ac39a3c6,91abc61a9969b5c6
Normal-1157.png,Normal,Normal/images/Normal-1157.png,Normal/masks/Normal-1157.png,True,G09783,train,8c2e72d0a69dcf190d1683871136723b,d07cc71b9d4f18c4
Normal-1158.png,Normal,Normal/images/Normal-1158.png,Normal/masks/Normal-1158.png,True,G09784,train,d3b0c7e2824531be698482dafd1e0708,946e8f1acd1cc297
Normal-1159.png,Normal,Normal/images/Normal-1159.png,Normal/masks/Normal-1159.png,True,G09785,train,20d030dd8ae754b80dc83d789569c82b,913d963a999d3994
Normal-116.png,Normal,Normal/images/Normal-116.png,Normal/masks/Normal-116.png,True,G09786,test,96149da06050a1fcc4320582068144b7,c03eef1e4c591d19
Normal-1160.png,Normal,Normal/images/Normal-1160.png,Normal/masks/Normal-1160.png,True,G09787,train,5d4df8546501bf8aab90aa0093653c82,9023c21efc7839f3
Normal-1161.png,Normal,Normal/images/Normal-1161.png,Normal/masks/Normal-1161.png,True,G09788,train,22e71b6fc407a1b24bee04fdb7a025ef,902de63af9693d48
Normal-1162.png,Normal,Normal/images/Normal-1162.png,Normal/masks/Normal-1162.png,True,G09789,train,57950348f39727f612bc6d7bc9709f32,907b565e190939bd
Normal-1163.png,Normal,Normal/images/Normal-1163.png,Normal/masks/Normal-1163.png,True,G09790,train,6dc523cd9288d946951de55b0a540f43,913dc71a984939f9
Normal-1164.png,Normal,Normal/images/Normal-1164.png,Normal/masks/Normal-1164.png,True,G09791,train,5f6b3de53b604aa35162bf787e283220,916cc61ecd4d4ccd
Normal-1165.png,Normal,Normal/images/Normal-1165.png,Normal/masks/Normal-1165.png,True,G09792,train,59572b23ae476bc32a91e0d01228810e,d16cc339bd783898
Normal-1166.png,Normal,Normal/images/Normal-1166.png,Normal/masks/Normal-1166.png,True,G09793,val,22e9ce9e2ddb74d3156799ba49bf0fce,d46c872f2d4d2ccc
Normal-1167.png,Normal,Normal/images/Normal-1167.png,Normal/masks/Normal-1167.png,True,G09794,val,1e5e788104eaf6b354f6e737c6c169ec,d07cc73fad2c2c44
Normal-1168.png,Normal,Normal/images/Normal-1168.png,Normal/masks/Normal-1168.png,True,G09795,test,5f710b7bd0c9269e1f8d857548e312ed,d46eaf2b0f2e2846
Normal-1169.png,Normal,Normal/images/Normal-1169.png,Normal/masks/Normal-1169.png,True,G09796,test,e1ad18bcdd1d8d129a791da053c8ce34,d46cc71ec92d6ca4
Normal-117.png,Normal,Normal/images/Normal-117.png,Normal/masks/Normal-117.png,True,G09797,train,684119c20ffad430e155212473f3ea05,902dc31bd9693dd1
Normal-1170.png,Normal,Normal/images/Normal-1170.png,Normal/masks/Normal-1170.png,True,G09798,train,353c1a9e49441444ba5c3154c6f829e5,d5e7831bad2c3825
Normal-1171.png,Normal,Normal/images/Normal-1171.png,Normal/masks/Normal-1171.png,True,G09799,train,90af0b011eabbb2a71fd93f8e50abc92,987fc61e4a4b1d45
Normal-1172.png,Normal,Normal/images/Normal-1172.png,Normal/masks/Normal-1172.png,True,G09800,train,1137d7bc53fd906127c25a7840f82f9e,c56fc30bc919d2c3
Normal-1173.png,Normal,Normal/images/Normal-1173.png,Normal/masks/Normal-1173.png,True,G09801,train,19a59f6d2f67bb55b36a59d11e621d86,902bc67a79693969
Normal-1174.png,Normal,Normal/images/Normal-1174.png,Normal/masks/Normal-1174.png,True,G09802,train,fb08280be7f92004bb34377020d13229,99bfc71a8c581dc4
Normal-1175.png,Normal,Normal/images/Normal-1175.png,Normal/masks/Normal-1175.png,True,G09803,train,38bde7c435b50ee67ffce86df9487e48,d42dc71fcd0d2cc4
Normal-1176.png,Normal,Normal/images/Normal-1176.png,Normal/masks/Normal-1176.png,True,G09804,val,6d2b0e34641667f11bc0efa58ddaa9f3,812bc61adc5b58dd
Normal-1177.png,Normal,Normal/images/Normal-1177.png,Normal/masks/Normal-1177.png,True,G09805,val,a0082231be034eae588d308d5c8d27ee,d56ec34fec6028e4
Normal-1178.png,Normal,Normal/images/Normal-1178.png,Normal/masks/Normal-1178.png,True,G09806,val,405e1d6ea16ad0c2cda38ea583e6885a,9569cc4bcbc75292
Normal-1179.png,Normal,Normal/images/Normal-1179.png,Normal/masks/Normal-1179.png,True,G09807,train,7b802452081b7dd93d5f87dd1f1443e9,816ac65ad85939db
Normal-118.png,Normal,Normal/images/Normal-118.png,Normal/masks/Normal-118.png,True,G09808,test,ed53b9e4d3ed0e21fcedb72fe2d64b8b,c060c779e9f46ce1
Normal-1180.png,Normal,Normal/images/Normal-1180.png,Normal/masks/Normal-1180.png,True,G09809,train,0f74438516504aaabb6a5cf12e1da536,91a8c61ac36bb86f
Normal-1181.png,Normal,Normal/images/Normal-1181.png,Normal/masks/Normal-1181.png,True,G09810,train,6ef5f6ccbd06ffb8f957d5218d8ff8f2,91e8863ace3a79c5
Normal-1182.png,Normal,Normal/images/Normal-1182.png,Normal/masks/Normal-1182.png,True,G09811,train,1d5ef5d82d9705117e2a5f1850daea86,912bc63ae87939c9
Normal-1183.png,Normal,Normal/images/Normal-1183.png,Normal/masks/Normal-1183.png,True,G09812,test,ee7ccfdce708b5e67393e51492fbf2a0,956cc31acb2f7834
Normal-1184.png,Normal,Normal/images/Normal-1184.png,Normal/masks/Normal-1184.png,True,G09813,train,d17c3246b0ed79f33a39d3a20615d828,91fcc63ab9293869
Normal-1185.png,Normal,Normal/images/Normal-1185.png,Normal/masks/Normal-1185.png,True,G09814,train,7d21937e074657eb8bb5d6d171789d27,993fc61a9f4b1849
Normal-1186.png,Normal,Normal/images/Normal-1186.png,Normal/masks/Normal-1186.png,True,G09815,train,e5af6e132e13093deebbe5ba6d2a9038,907ec65adc591dc4
Normal-1187.png,Normal,Normal/images/Normal-1187.png,Normal/masks/Normal-1187.png,True,G09816,test,dd8fbf2af0961694992d38fa28c12f8f,d47dc71a93833991
Normal-1188.png,Normal,Normal/images/Normal-1188.png,Normal/masks/Normal-1188.png,True,G09817,train,2d72daf8c28841336f9d68d6669db9a5,903be71ade4d1991
Normal-1189.png,Normal,Normal/images/Normal-1189.png,Normal/masks/Normal-1189.png,True,G09818,train,cce0ef91f1e8a1bf9a60ae0bbfd9d339,943e8b1e5c5c1cd6
Normal-119.png,Normal,Normal/images/Normal-119.png,Normal/masks/Normal-119.png,True,G09819,train,f3f1c7a7bf75322a027e6e70028f8d92,d029e31bf9591993
Normal-1190.png,Normal,Normal/images/Normal-1190.png,Normal/masks/Normal-1190.png,True,G09820,train,cd8b547d813a3a5b55bef33899fea5a5,91be8e1a9d4b19c5
Normal-1191.png,Normal,Normal/images/Normal-1191.png,Normal/masks/Normal-1191.png,True,G09821,test,dca6717a728b978543f9cf0b80c4d08e,d16fce4a1d4133c3
Normal-1192.png,Normal,Normal/images/Normal-1192.png,Normal/masks/Normal-1192.png,True,G09822,train,203d301d1233f66b9a06de647cc34e06,d929e61ed62aba48
Normal-1193.png,Normal,Normal/images/Normal-1193.png,Normal/masks/Normal-1193.png,True,G09823,train,364ca545ab1bb3faa85f994f1b2f7687,982dc21ecf4f3dc0
Normal-1194.png,Normal,Normal/images/Normal-1194.png,Normal/masks/Normal-1194.png,True,G09824,train,a1a317da0053a8098d5a8b61c12daddf,d46ec75bcd252886
Normal-1195.png,Normal,Normal/images/Normal-1195.png,Normal/masks/Normal-1195.png,True,G09825,train,5b18ba32080a123a2d54735534df2677,9128ce38c97999d7
Normal-1196.png,Normal,Normal/images/Normal-1196.png,Normal/masks/Normal-1196.png,True,G09826,val,1fd4e75bdcbb58fdd4b08c60efe8a476,8869c75af8793949
Normal-1197.png,Normal,Normal/images/Normal-1197.png,Normal/masks/Normal-1197.png,True,G09827,test,3c1da6fb1a986dc5f3b3b43aec814c05,d066c71bcd2d89c6
Normal-1198.png,Normal,Normal/images/Normal-1198.png,Normal/masks/Normal-1198.png,True,G09828,train,47ca5e2406222624c780b0458d21d16a,816ace5a5d4d585d
Normal-1199.png,Normal,Normal/images/Normal-1199.png,Normal/masks/Normal-1199.png,True,G09829,train,6feda817e1f42ebb45736cdabee4b71d,d47cc32fbd682434
Normal-12.png,Normal,Normal/images/Normal-12.png,Normal/masks/Normal-12.png,True,G09830,train,15bf509e8e506e1b0861522e8692f57d,c96d861b19545f59
Normal-120.png,Normal,Normal/images/Normal-120.png,Normal/masks/Normal-120.png,True,G09831,train,fd072fdcdc3049ebbc9fdbee390b22c3,d12bc25b1e68795c
Normal-1200.png,Normal,Normal/images/Normal-1200.png,Normal/masks/Normal-1200.png,True,G09832,train,2c1b7def1241ec7712a5afaf5e622486,9128da7a5c5d1d47
Normal-1201.png,Normal,Normal/images/Normal-1201.png,Normal/masks/Normal-1201.png,True,G09833,train,4048ac3409160bc16b4b0514b6a32ac0,916f4e1a05595cd7
Normal-1202.png,Normal,Normal/images/Normal-1202.png,Normal/masks/Normal-1202.png,True,G09834,train,6a7f4697768a2c275a2dc51500213d2c,9039e71ebe431e34
Normal-1203.png,Normal,Normal/images/Normal-1203.png,Normal/masks/Normal-1203.png,True,G09835,val,36626672bcbcacbd038a2090e7189cda,912bc63af87939c1
Normal-1204.png,Normal,Normal/images/Normal-1204.png,Normal/masks/Normal-1204.png,True,G09836,test,a47a2dac677dc418217fc33ec7587aed,9139ce18ddcb1aca
Normal-1205.png,Normal,Normal/images/Normal-1205.png,Normal/masks/Normal-1205.png,True,G09837,val,e0672ef63a2d5e6396e5f060e31690ff,d476c35bed0128f4
Normal-1206.png,Normal,Normal/images/Normal-1206.png,Normal/masks/Normal-1206.png,True,G09838,test,8b5c0ee2b081bbbd1426c84b8c3d2349,d02dc73abc493c9c
Normal-1207.png,Normal,Normal/images/Normal-1207.png,Normal/masks/Normal-1207.png,True,G09839,train,0fa86f516cb9ebdb9a369aeeb1701012,d468c33be9696ce0
Normal-1208.png,Normal,Normal/images/Normal-1208.png,Normal/masks/Normal-1208.png,True,G09840,train,4a509f6e77129b11073344d1434408ea,d16cc74ae9e529c4
Normal-1209.png,Normal,Normal/images/Normal-1209.png,Normal/masks/Normal-1209.png,True,G09841,train,edeb6ee9579e180c7636def3956279d9,913ec61adc5d18cd
Normal-121.png,Normal,Normal/images/Normal-121.png,Normal/masks/Normal-121.png,True,G09842,train,718580bccbf9361dca003f020238774e,916ece7a4d4149c7
Normal-1210.png,Normal,Normal/images/Normal-1210.png,Normal/masks/Normal-1210.png,True,G09843,train,0b41668ca7cd57ffe2382f644bb958eb,912bc75a98c97939
Normal-1211.png,Normal,Normal/images/Normal-1211.png,Normal/masks/Normal-1211.png,True,G09844,train,62fab1dcc964428b6d3906e793df5a4f,dd6e831bc53c4625
Normal-1212.png,Normal,Normal/images/Normal-1212.png,Normal/masks/Normal-1212.png,True,G09845,train,aa426f7de64b02b243a80be923de3ec2,d03dc31bdc4c3cd8
Normal-1213.png,Normal,Normal/images/Normal-1213.png,Normal/masks/Normal-1213.png,True,G09846,train,5acec4890d69350f4a9e734ea7da2881,d03a465f1ed8699c
Normal-1214.png,Normal,Normal/images/Normal-1214.png,Normal/masks/Normal-1214.png,True,G09847,train,93a7da7b17054ee40626d05b7c5902bd,d42fe31b8d0d3c4c
Normal-1215.png,Normal,Normal/images/Normal-1215.png,Normal/masks/Normal-1215.png,True,G09848,train,794afcf4f1d7e55532f6258c946ff44f,8174c61acf6d31cd
Normal-1216.png,Normal,Normal/images/Normal-1216.png,Normal/masks/Normal-1216.png,True,G09849,train,a45a87524a39254cf550ce2eb69354a3,916fcf1a890d49c7
Normal-1217.png,Normal,Normal/images/Normal-1217.png,Normal/masks/Normal-1217.png,True,G09850,train,c4a110ac5c2e8a3916ddc19bd0bf59fb,906dc71ecd4919cd
Normal-1218.png,Normal,Normal/images/Normal-1218.png,Normal/masks/Normal-1218.png,True,G09851,train,e6a7ae7d8c18a463de49563a42d3b369,81d4c61ac987f3c7
Normal-1219.png,Normal,Normal/images/Normal-1219.png,Normal/masks/Normal-1219.png,True,G09852,val,bd93f3f8e1f71249217ccc8b57fcbf51,d47e870bc1a568e5
Normal-122.png,Normal,Normal/images/Normal-122.png,Normal/masks/Normal-122.png,True,G09853,val,78ee66d6a1ccb5ffa77284b0e294764e,913cc61ab9473fc2
Normal-1220.png,Normal,Normal/images/Normal-1220.png,Normal/masks/Normal-1220.png,True,G09854,val,57c7a03b04c4d835b119ea143362805a,d02fc72fcd090cc7
Normal-1221.png,Normal,Normal/images/Normal-1221.png,Normal/masks/Normal-1221.png,True,G09855,test,ae17bfe279a99c5c0970d2bf9abaa1a9,d434c30f9d693ce4
Normal-1222.png,Normal,Normal/images/Normal-1222.png,Normal/masks/Normal-1222.png,True,G09856,test,bc9fd10c54d77b022516248b2c85f044,8029c65afa7a39d9
Normal-1223.png,Normal,Normal/images/Normal-1223.png,Normal/masks/Normal-1223.png,True,G09857,train,8c81b26d5d4e0141ad62ebccd83dd0fe,d46fc71bc92964c4
Normal-1224.png,Normal,Normal/images/Normal-1224.png,Normal/masks/Normal-1224.png,True,G09858,train,e23199cf792d550fb5ec7f57e23cc3ff,d03fc61f9cc91cc2
Normal-1225.png,Normal,Normal/images/Normal-1225.png,Normal/masks/Normal-1225.png,True,G09859,train,d2697d9c08ce87a30accaedaffe8c14a,8029c75abac939bd
Normal-1226.png,Normal,Normal/images/Normal-1226.png,Normal/masks/Normal-1226.png,True,G09860,train,dd6d2501b588f7569fc5c27801d7c804,813dc61abd693d4c
Normal-1227.png,Normal,Normal/images/Normal-1227.png,Normal/masks/Normal-1227.png,True,G09861,train,e6eadd9bc2d3fd3251561046b0499c00,cd6c061a454f7c4f
Normal-1228.png,Normal,Normal/images/Normal-1228.png,Normal/masks/Normal-1228.png,True,G09862,train,65f977390fcc5999b16d28b1d531a783,d476875b8d2c6c86
Normal-1229.png,Normal,Normal/images/Normal-1229.png,Normal/masks/Normal-1229.png,True,G09863,train,1e8b07a3f3d3f3a8a9110c8b50b49ac6,946fe33a0f4b1d44
Normal-123.png,Normal,Normal/images/Normal-123.png,Normal/masks/Normal-123.png,True,G09864,train,171d5fc4d02db631a01bce6fb7c5a349,912fc21ad86b97c6
Normal-1230.png,Normal,Normal/images/Normal-1230.png,Normal/masks/Normal-1230.png,True,G09865,train,14d991f0dc90790bc9ffcc6c258c4253,d47e870f852d2ce4
Normal-1231.png,Normal,Normal/images/Normal-1231.png,Normal/masks/Normal-1231.png,True,G09866,train,eced66f5a4286376b0d9a5f941037be3,912dc61abc4b39c7
Normal-1232.png,Normal,Normal/images/Normal-1232.png,Normal/masks/Normal-1232.png,True,G09867,train,9cdf6eb13ac7b1cd2f13157bc124ce88,9029c73af86939dc
Normal-1233.png,Normal,Normal/images/Normal-1233.png,Normal/masks/Normal-1233.png,True,G09868,train,1f05e45b470f9b2e365c056732126c76,917cc63ab96d38b0
Normal-1234.png,Normal,Normal/images/Normal-1234.png,Normal/masks/Normal-1234.png,True,G09869,train,4d5367587e13b4a5d394bdc116a8f406,802ac71ade493df9
Normal-1235.png,Normal,Normal/images/Normal-1235.png,Normal/masks/Normal-1235.png,True,G09870,val,539e0d9f845e28075efa3c568fa74ed7,9829e21afb783c69
Normal-1236.png,Normal,Normal/images/Normal-1236.png,Normal/masks/Normal-1236.png,True,G09871,test,ac3d452549e23a9b1e2a004f5eda122e,946e831b8bbd68e4
Normal-1237.png,Normal,Normal/images/Normal-1237.png,Normal/masks/Normal-1237.png,True,G09872,test,9034e8a68c3858b6402cfb6b62b1d46e,982cd71a8f581d5d
Normal-1238.png,Normal,Normal/images/Normal-1238.png,Normal/masks/Normal-1238.png,True,G09873,train,4ffa75ad70d157cf4e0a5f76cea53c18,947f8f0f8d0d0cc3
Normal-1239.png,Normal,Normal/images/Normal-1239.png,Normal/masks/Normal-1239.png,True,G09874,train,1f312d57e5da3f3bd5271622339e9fec,d12dc71a9cc919c7
Normal-124.png,Normal,Normal/images/Normal-124.png,Normal/masks/Normal-124.png,True,G09875,val,deddcf387ae22e4ed8544ba6a4369e41,917cce5a8d4d1ccc
Normal-1240.png,Normal,Normal/images/Normal-1240.png,Normal/masks/Normal-1240.png,True,G09876,train,09c85d4879019c946f111f5ff0406628,8129c75af8693979
Normal-1241.png,Normal,Normal/images/Normal-1241.png,Normal/masks/Normal-1241.png,True,G09877,train,55aafacba184f2fb6e99b3ed36ee722c,9129c61afce939c5
Normal-1242.png,Normal,Normal/images/Normal-1242.png,Normal/masks/Normal-1242.png,True,G09878,train,15b63459d05d31cb152a354353058cae,957c875a8d3c3896
Normal-1243.png,Normal,Normal/images/Normal-1243.png,Normal/masks/Normal-1243.png,True,G09879,val,9f68320ed05bb4f4c1a0bdfa729ee5ff,986fc71a0b4d3c4d
Normal-1244.png,Normal,Normal/images/Normal-1244.png,Normal/masks/Normal-1244.png,True,G09880,test,d9dd614868a63a51d99d52a01972f77a,913ac25a5e4a7d3c
Normal-1245.png,Normal,Normal/images/Normal-1245.png,Normal/masks/Normal-1245.png,True,G09881,train,ac35894e58d3650063968f3e1bce0077,916aee7a5d491945
Normal-1246.png,Normal,Normal/images/Normal-1246.png,Normal/masks/Normal-1246.png,True,G09882,train,2b25b83244089c7e4fe388780404c5a4,917dc61a996939c5
Normal-1247.png,Normal,Normal/images/Normal-1247.png,Normal/masks/Normal-1247.png,True,G09883,val,6ae831ecf36c6b4d89067df2503bbe6c,912bc70a9ee959c5
Normal-1248.png,Normal,Normal/images/Normal-1248.png,Normal/masks/Normal-1248.png,True,G09884,train,6fb6662d7de64feac7ec21a4da64d79e,d03bc29afcc626a6
Normal-1249.png,Normal,Normal/images/Normal-1249.png,Normal/masks/Normal-1249.png,True,G09885,train,9d264af3aa9f2969917f9676ddbbfb57,9139c639f94939d1
Normal-125.png,Normal,Normal/images/Normal-125.png,Normal/masks/Normal-125.png,True,G09886,train,079765b42969a0462bd1e04f19acf69e,d86f475e4d491c4a
Normal-1250.png,Normal,Normal/images/Normal-1250.png,Normal/masks/Normal-1250.png,True,G09887,train,ef31cf73c8593cfe64bdca5f6e386a5f,913bd21ad8483df3
Normal-1251.png,Normal,Normal/images/Normal-1251.png,Normal/masks/Normal-1251.png,True,G09888,train,f6d91f2acce14ed595b05a99fabe0397,916ca75a8b2f384d
Normal-1252.png,Normal,Normal/images/Normal-1252.png,Normal/masks/Normal-1252.png,True,G09889,train,3bc599d38e364106a3611cb3e3244832,d17fe63e8d0d0c84
Normal-1253.png,Normal,Normal/images/Normal-1253.png,Normal/masks/Normal-1253.png,True,G09890,train,120169411ee0587a6dded5000277c3bf,d57f871b8d0d0c4c
Normal-1254.png,Normal,Normal/images/Normal-1254.png,Normal/masks/Normal-1254.png,True,G09891,train,cb7d237482f3438c0a9f5c5e16dbe95a,d03cc73ecd1c38cc
Normal-1255.png,Normal,Normal/images/Normal-1255.png,Normal/masks/Normal-1255.png,True,G09892,val,5a052ed57d2dab50f1f947203d85bf2c,d07c863a8d3d6cc6
Normal-1256.png,Normal,Normal/images/Normal-1256.png,Normal/masks/Normal-1256.png,True,G09893,val,80eb0e03d4a35ec1097f1a83a6e3e00d,903af21adc511dd7
Normal-1257.png,Normal,Normal/images/Normal-1257.png,Normal/masks/Normal-1257.png,True,G09894,test,253284cde5ea14409c432980f90f0fde,9129c63af97d3819
Normal-1258.png,Normal,Normal/images/Normal-1258.png,Normal/masks/Normal-1258.png,True,G09895,test,ddae1e465cfce3b11fceaf850ab4f3ac,d074a73aa92b39e4
Normal-1259.png,Normal,Normal/images/Normal-1259.png,Normal/masks/Normal-1259.png,True,G09896,test,d4137de727367444e82793b8106e22b6,9029e63afa693d49
Normal-126.png,Normal,Normal/images/Normal-126.png,Normal/masks/Normal-126.png,True,G09897,train,4cc384639297203f09790b28fad29085,916cc678ed6d4986
Normal-1260.png,Normal,Normal/images/Normal-1260.png,Normal/masks/Normal-1260.png,True,G09898,train,4738177c7cf229fb5317a4e5ce641127,d13dc73e8929384d
Normal-1261.png,Normal,Normal/images/Normal-1261.png,Normal/masks/Normal-1261.png,True,G09899,train,056eabf737d07c0a547424e69ece3040,906cc71acd791cc7
Normal-1262.png,Normal,Normal/images/Normal-1262.png,Normal/masks/Normal-1262.png,True,G09900,train,390710430dba67638ec13a29ae9e4417,d131c61aba693979
Normal-1263.png,Normal,Normal/images/Normal-1263.png,Normal/masks/Normal-1263.png,True,G09901,train,b7df077c16ec5f191e891ab97faceda8,812fc67a98c939d5
Normal-1264.png,Normal,Normal/images/Normal-1264.png,Normal/masks/Normal-1264.png,True,G09902,train,64424af3d7a608595e6380bf982aba85,9139c61af9c939c9
Normal-1265.png,Normal,Normal/images/Normal-1265.png,Normal/masks/Normal-1265.png,True,G09903,train,780e2894b558ad50af6d6339d602b6c5,c014c71bed4d3de4
Normal-1266.png,Normal,Normal/images/Normal-1266.png,Normal/masks/Normal-1266.png,True,G09904,train,a41964969fd133791afda07d71a2efe2,c164c33ced3cccc5
Normal-1267.png,Normal,Normal/images/Normal-1267.png,Normal/masks/Normal-1267.png,True,G09905,train,12fdb658ec15608a66ff049557de9b72,912bc65b7c496939
Normal-1268.png,Normal,Normal/images/Normal-1268.png,Normal/masks/Normal-1268.png,True,G09906,train,27417b2e3001eec70d889b6deb9ce0ed,906dc71ae96f28c5
Normal-1269.png,Normal,Normal/images/Normal-1269.png,Normal/masks/Normal-1269.png,True,G09907,train,320fb8201c6f4495783c6b03484bcd43,816bc67a78797911
Normal-127.png,Normal,Normal/images/Normal-127.png,Normal/masks/Normal-127.png,True,G09908,test,28a7873528dc1c24f99560db052e2ff1,916cc658cd6d38c7
Normal-1270.png,Normal,Normal/images/Normal-1270.png,Normal/masks/Normal-1270.png,True,G09909,train,5c9f68ad3c3478be6bccdd9b1cd1b032,9168ae3aed69386c
Normal-1271.png,Normal,Normal/images/Normal-1271.png,Normal/masks/Normal-1271.png,True,G09910,train,046fa0d23292f30bc87dfd025beed006,997ac65a5d5d1905
Normal-1272.png,Normal,Normal/images/Normal-1272.png,Normal/masks/Normal-1272.png,True,G09911,test,00eca591cb97166f1d4df34b7020eb01,d07ea73fad242c84
Normal-1273.png,Normal,Normal/images/Normal-1273.png,Normal/masks/Normal-1273.png,True,G09912,train,e27769236d75f1e78910644e649d8093,9133c61af465b6c6
Normal-1274.png,Normal,Normal/images/Normal-1274.png,Normal/masks/Normal-1274.png,True,G09913,val,f9d85f25a353073eaba8398cd9309d7a,916dc63a8d6d3c4c
Normal-1275.png,Normal,Normal/images/Normal-1275.png,Normal/masks/Normal-1275.png,True,G09914,val,e4670693efa52434a1a2f02134810f86,d46fc30fe56460c6
Normal-1276.png,Normal,Normal/images/Normal-1276.png,Normal/masks/Normal-1276.png,True,G09915,val,1fd97998c421da1d309b4e6769f8e37b,d464c61fed6c2cc4
Normal-1277.png,Normal,Normal/images/Normal-1277.png,Normal/masks/Normal-1277.png,True,G09916,test,e07f1227fbae083050e41ed3e7d21096,812bc21af86939fb
Normal-1278.png,Normal,Normal/images/Normal-1278.png,Normal/masks/Normal-1278.png,True,G09917,train,6b54ccbf1b72543316ef7f398a18e1a9,c5fea31b2d6c18c4
Normal-1279.png,Normal,Normal/images/Normal-1279.png,Normal/masks/Normal-1279.png,True,G09918,train,2b616a8e5406c36765147873aa4cbaaa,c46ec70bed6d68c0
Normal-128.png,Normal,Normal/images/Normal-128.png,Normal/masks/Normal-128.png,True,G09919,train,d9088ff8b51f731cdc68d843cc743ca8,d02bda38f85a1ad6
Normal-1280.png,Normal,Normal/images/Normal-1280.png,Normal/masks/Normal-1280.png,True,G09920,train,f9de1f14953a4b1326db2551382c93a2,d039c35adf4b1d48
Normal-1281.png,Normal,Normal/images/Normal-1281.png,Normal/masks/Normal-1281.png,True,G09921,train,7b3bd065c26a82d42fba41f3b6da2f5b,d0f6e61a6c0d64cd
Normal-1282.png,Normal,Normal/images/Normal-1282.png,Normal/masks/Normal-1282.png,True,G09922,val,d91421eb3c40b3327ecfc204b5752df1,d07cc61efc2d3c44
Normal-1283.png,Normal,Normal/images/Normal-1283.png,Normal/masks/Normal-1283.png,True,G09923,train,2465e4d39c8ee5feddc677df82179ba1,9178de7a595b1903
Normal-1284.png,Normal,Normal/images/Normal-1284.png,Normal/masks/Normal-1284.png,True,G09924,train,e284f304a1b2cfae94d0f1537d406e1d,9135c61a9d493df1
Normal-1285.png,Normal,Normal/images/Normal-1285.png,Normal/masks/Normal-1285.png,True,G09925,train,bc37b7009cf26f85ec64243b729d337e,912bc61a9cf83db1
Normal-1286.png,Normal,Normal/images/Normal-1286.png,Normal/masks/Normal-1286.png,True,G09926,train,66e5bca580484b0ae6aecc99d589804f,d43e830b9ec63cb4
Normal-1287.png,Normal,Normal/images/Normal-1287.png,Normal/masks/Normal-1287.png,True,G09927,train,67c4d57bb895876793259dd58443f32e,8075871aca6d79e5
Normal-1288.png,Normal,Normal/images/Normal-1288.png,Normal/masks/Normal-1288.png,True,G09928,train,3e1fccd0e95e5dd775243b4432512d5f,c139e31ad8693979
Normal-1289.png,Normal,Normal/images/Normal-1289.png,Normal/masks/Normal-1289.png,True,G09929,train,b362388f94163dcb233ac6dc6205754e,9521e20bed7c3ce4
Normal-129.png,Normal,Normal/images/Normal-129.png,Normal/masks/Normal-129.png,True,G09930,val,731e100bb31ec8f1dd5486dabe74e4e7,9064875acd2dedc6
Normal-1290.png,Normal,Normal/images/Normal-1290.png,Normal/masks/Normal-1290.png,True,G09931,train,d2ccd1869f33331c1fb0828dad18e97d,d47f873b8d0d20c6
Normal-1291.png,Normal,Normal/images/Normal-1291.png,Normal/masks/Normal-1291.png,True,G09932,train,b04452c2419002234155307ccbd44c0f,916bc61af8493dd2
Normal-1292.png,Normal,Normal/images/Normal-1292.png,Normal/masks/Normal-1292.png,True,G09933,val,7f557981d86042f9fa61d5d2d201386f,d06ec71fbdc52824
Normal-1293.png,Normal,Normal/images/Normal-1293.png,Normal/masks/Normal-1293.png,True,G09934,test,5e9e8d4d056caf63ad52e1462e4f1add,997f461a5e4b3849
Normal-1294.png,Normal,Normal/images/Normal-1294.png,Normal/masks/Normal-1294.png,True,G09935,test,aab0b4c93fcc4bee7791a64800fedfbe,d04ac65af1c339f2
Normal-1295.png,Normal,Normal/images/Normal-1295.png,Normal/masks/Normal-1295.png,True,G09936,test,510941bb0c4251bd5ae4cff0f1082ef7,983fc21a4b1d5c4f
Normal-1296.png,Normal,Normal/images/Normal-1296.png,Normal/masks/Normal-1296.png,True,G09937,train,85e756c773c3867ae61a7087c9020a5d,916fc71a8b293c6c
Normal-1297.png,Normal,Normal/images/Normal-1297.png,Normal/masks/Normal-1297.png,True,G09938,train,ed898a4f47dba4ae1d0a4bcf15b77d44,81ac871b174f394f
Normal-1298.png,Normal,Normal/images/Normal-1298.png,Normal/masks/Normal-1298.png,True,G09939,train,d2caa2a04252a9fc32694ac7f778d449,997e4612474e786d
Normal-1299.png,Normal,Normal/images/Normal-1299.png,Normal/masks/Normal-1299.png,True,G09940,train,5fafdc8f4e337f404c293ad5914f0c58,902de63ecb2d386c
Normal-13.png,Normal,Normal/images/Normal-13.png,Normal/masks/Normal-13.png,True,G09941,train,60ea02999ae1c45aabc020273c7195c4,dc2cc60f0c4c3fec
Normal-130.png,Normal,Normal/images/Normal-130.png,Normal/masks/Normal-130.png,True,G09942,train,78c7b64a27de0f2c4d935d3ab6e3241a,c0feef2d2cc02a99
Normal-1300.png,Normal,Normal/images/Normal-1300.png,Normal/masks/Normal-1300.png,True,G09943,train,fd36a70c0a7ea0bf82d6d17b0b02b47e,91eec71acd0d49c5
Normal-1301.png,Normal,Normal/images/Normal-1301.png,Normal/masks/Normal-1301.png,True,G09944,train,0a7cc305dcec3604aa6ea5f3541d6d8a,912bc23ad85b39d3
Normal-1302.png,Normal,Normal/images/Normal-1302.png,Normal/masks/Normal-1302.png,True,G09945,train,0c4166e35f807bea684db4b137e155a6,902dc61acf691dcd
Normal-1303.png,Normal,Normal/images/Normal-1303.png,Normal/masks/Normal-1303.png,True,G09946,train,2d8f5bacdf7c2a4c70862814f45d6fe6,d5fcc75ae54920c4
Normal-1304.png,Normal,Normal/images/Normal-1304.png,Normal/masks/Normal-1304.png,True,G09947,val,2c8d31955127b1b036a0384f1f9d0644,903fab1a0d591dd5
Normal-1305.png,Normal,Normal/images/Normal-1305.png,Normal/masks/Normal-1305.png,True,G09948,test,2dbf3b2c1b93620eba5340bf84257ae9,846cc74bcb4f48e5
Normal-1306.png,Normal,Normal/images/Normal-1306.png,Normal/masks/Normal-1306.png,True,G09949,train,2b8273e8f5a4e224b833e231b907d97c,902ee33aec6b38e4
Normal-1307.png,Normal,Normal/images/Normal-1307.png,Normal/masks/Normal-1307.png,True,G09950,train,c67d6004f370a42d87466bb5495f3fd1,91bac61a9c1ab5d6
Normal-1308.png,Normal,Normal/images/Normal-1308.png,Normal/masks/Normal-1308.png,True,G09951,train,fb116aa682b20fce5cc9ecfcd2eb8f77,812bc65a7849797b
Normal-1309.png,Normal,Normal/images/Normal-1309.png,Normal/masks/Normal-1309.png,True,G09952,train,738fb4071b477662384f30ecb03127aa,986dc71ecf498cc1
Normal-131.png,Normal,Normal/images/Normal-131.png,Normal/masks/Normal-131.png,True,G09953,val,6882fd9e94a55fe0b00b4b18e90a9cc6,956f8f0a094d5cce
Normal-1310.png,Normal,Normal/images/Normal-1310.png,Normal/masks/Normal-1310.png,True,G09954,val,9ea465333260dc6d15372d889655c0b7,90ade61aeb2d386c
Normal-1311.png,Normal,Normal/images/Normal-1311.png,Normal/masks/Normal-1311.png,True,G09955,test,05fbef43205e4e08927e922326d28609,d0bdc31b9cc438b6
Normal-1312.png,Normal,Normal/images/Normal-1312.png,Normal/masks/Normal-1312.png,True,G09956,train,770e0c87624a8b62c639e89767893cf3,812fc61ab9c93d95
Normal-1313.png,Normal,Normal/images/Normal-1313.png,Normal/masks/Normal-1313.png,True,G09957,test,2e11547e67af9a2f777f107e99a81d96,91adc63aea693067
Normal-1314.png,Normal,Normal/images/Normal-1314.png,Normal/masks/Normal-1314.png,True,G09958,train,1f3fa456a18fc3eb875e72ca022627cb,d034c33bdd6d3cc0
Normal-1315.png,Normal,Normal/images/Normal-1315.png,Normal/masks/Normal-1315.png,True,G09959,train,543df5983d17f9a8679e6f0355a9c9c0,d166c73b856d2cc4
Normal-1316.png,Normal,Normal/images/Normal-1316.png,Normal/masks/Normal-1316.png,True,G09960,train,90b94b3d35a7af70887cf0abdc5e133a,d46dc35bcd2c4ce0
Normal-1317.png,Normal,Normal/images/Normal-1317.png,Normal/masks/Normal-1317.png,True,G09961,train,266a18281bdfefaa5cee2782f6a98533,986f871a174f1a4d
Normal-1318.png,Normal,Normal/images/Normal-1318.png,Normal/masks/Normal-1318.png,True,G09962,train,6d81bd730fafc656a1a2f5bba011ca18,803de71ede4b3868
Normal-1319.png,Normal,Normal/images/Normal-1319.png,Normal/masks/Normal-1319.png,True,G09963,train,6e202d7bc2b39243b67417d3d6eda358,903fc318d8399ccf
Normal-132.png,Normal,Normal/images/Normal-132.png,Normal/masks/Normal-132.png,True,G09964,train,d6c27935e856fcf91336661252015d71,d46e834bcd2c6cc6
Normal-1320.png,Normal,Normal/images/Normal-1320.png,Normal/masks/Normal-1320.png,True,G09965,train,bf91fa18aca79fc4b63ac18aae4ce07f,d57ec73e098568c4
Normal-1321.png,Normal,Normal/images/Normal-1321.png,Normal/masks/Normal-1321.png,True,G09966,train,b0ff3b65089cb2f0165233e373397598,d02de20bdc691de9
Normal-1322.png,Normal,Normal/images/Normal-1322.png,Normal/masks/Normal-1322.png,True,G09967,train,b3c68a1b691183cf8ecda988ecde0cc3,c16f474f45414dc7
Normal-1323.png,Normal,Normal/images/Normal-1323.png,Normal/masks/Normal-1323.png,True,G09968,train,cf67e676c58e8fe2186cc2b9fce3d40c,d03dc31b9d6938d8
Normal-1324.png,Normal,Normal/images/Normal-1324.png,Normal/masks/Normal-1324.png,True,G09969,train,ab87cac98b91c6d0eb493ce2729a5bc6,d02bc21f5f693c68
Normal-1325.png,Normal,Normal/images/Normal-1325.png,Normal/masks/Normal-1325.png,True,G09970,val,14d285892c229d085d0222aeba509851,d02ce73be9293c4c
Normal-1326.png,Normal,Normal/images/Normal-1326.png,Normal/masks/Normal-1326.png,True,G09971,val,ad6c3d7c1dc83ef56bfdbe38e27a3f91,8139c61ab8cc39fd
Normal-1327.png,Normal,Normal/images/Normal-1327.png,Normal/masks/Normal-1327.png,True,G09972,test,5f89ef6d0d4b91004bf9f2b036e5df49,dd2cc70b6d4d2cc4
Normal-1328.png,Normal,Normal/images/Normal-1328.png,Normal/masks/Normal-1328.png,True,G09973,train,34ce44b8c9443ac505cea70302beea89,d074c61a93cf3da4
Normal-1329.png,Normal,Normal/images/Normal-1329.png,Normal/masks/Normal-1329.png,True,G09974,val,5793b03bac58c1d1b50fb74a5cd4c414,91ad8e1a090d7d5f
Normal-133.png,Normal,Normal/images/Normal-133.png,Normal/masks/Normal-133.png,True,G09975,train,5003cce5c1a7f2286ee15cd3b7a8ba6c,d17c8639bd4c1c9c
Normal-1330.png,Normal,Normal/images/Normal-1330.png,Normal/masks/Normal-1330.png,True,G09976,test,c5ed6bbceca3757b0994b713cce5d5cc,816de63ae96d6845
Normal-1331.png,Normal,Normal/images/Normal-1331.png,Normal/masks/Normal-1331.png,True,G09977,test,d6ad61855a87b2d36704386f731cb381,802bc279796939f9
Normal-1332.png,Normal,Normal/images/Normal-1332.png,Normal/masks/Normal-1332.png,True,G09978,train,9e4c20c514e3b3f18fc5ae413f812075,812bc63e98ca3df1
Normal-1333.png,Normal,Normal/images/Normal-1333.png,Normal/masks/Normal-1333.png,True,G09979,train,83524d054fa367b018ed2509751f80ed,d12cc63bdb4f2c60
Normal-1334.png,Normal,Normal/images/Normal-1334.png,Normal/masks/Normal-1334.png,True,G09980,train,8e47f319015a1b03fb48be8631d4a17a,d56e870f8d4d4cc4
Normal-1335.png,Normal,Normal/images/Normal-1335.png,Normal/masks/Normal-1335.png,True,G09981,train,41e1de78313dff34b52bd8ac6848502c,812b425f5ce939b9
Normal-1336.png,Normal,Normal/images/Normal-1336.png,Normal/masks/Normal-1336.png,True,G09982,train,1388215b969873cc28633d0de59fccad,913cce1ac9493dcd
Normal-1337.png,Normal,Normal/images/Normal-1337.png,Normal/masks/Normal-1337.png,True,G09983,train,e690958360688d7441a1b95123636e6f,d424c31b8f4f2de4
Normal-1338.png,Normal,Normal/images/Normal-1338.png,Normal/masks/Normal-1338.png,True,G09984,train,9a301a3a9ba287dca9ea260795f71580,906bce1a59591dce
Normal-1339.png,Normal,Normal/images/Normal-1339.png,Normal/masks/Normal-1339.png,True,G09985,train,658685afd61bdb6a5ee9101826d55ec8,d050c71e9b6d3c65
Normal-134.png,Normal,Normal/images/Normal-134.png,Normal/masks/Normal-134.png,True,G09986,train,7a630b429613818dd9cd61e4c014fcc9,9128ce18d94f1fc7
Normal-1340.png,Normal,Normal/images/Normal-1340.png,Normal/masks/Normal-1340.png,True,G09987,train,cce6e3ae0ca45273e59bb709558d6a4a,d46cc75bcd0c6cc4
Normal-1341.png,Normal,Normal/images/Normal-1341.png,Normal/masks/Normal-1341.png,True,G09988,train,1c46b61b1c0a59140f92af60ded28607,9c7d871a8b1b3c8c
Normal-1342.png,Normal,Normal/images/Normal-1342.png,Normal/masks/Normal-1342.png,True,G09989,train,b43edb6109d495dd1fb6493b2d02e481,d476c33fa5242cc6
Normal-1343.png,Normal,Normal/images/Normal-1343.png,Normal/masks/Normal-1343.png,True,G09990,train,6c043e82ff3d4effcf7a652f07f3801f,992e065a596d78ec
Normal-1344.png,Normal,Normal/images/Normal-1344.png,Normal/masks/Normal-1344.png,True,G09991,val,63b68f8d56cc2b94fd5ab65a4bea27bf,917cc75acd0d49c5
Normal-1345.png,Normal,Normal/images/Normal-1345.png,Normal/masks/Normal-1345.png,True,G09992,test,346efb19d21bf53bad8cd4afa908ce5a,946f871aad1c6c4d
Normal-1346.png,Normal,Normal/images/Normal-1346.png,Normal/masks/Normal-1346.png,True,G09993,val,2a46f3fe3595659f18d87bf886f048c8,811f471a594d79e5
Normal-1347.png,Normal,Normal/images/Normal-1347.png,Normal/masks/Normal-1347.png,True,G09994,val,905b358286142c378d378f1547fd1304,916f8f3a8f19494c
Normal-1348.png,Normal,Normal/images/Normal-1348.png,Normal/masks/Normal-1348.png,True,G09995,test,67812adaa9dd5eea45f3fad3d7282792,d12fc21b9cc93d92
Normal-1349.png,Normal,Normal/images/Normal-1349.png,Normal/masks/Normal-1349.png,True,G09996,test,5181256bdc3b849dc7d684fcc2cf56c6,993ace1a9c571696
Normal-135.png,Normal,Normal/images/Normal-135.png,Normal/masks/Normal-135.png,True,G09997,train,ee2099bfb19cafcf5ce001b4ffa8c46c,917dc71acd2d388c
Normal-1350.png,Normal,Normal/images/Normal-1350.png,Normal/masks/Normal-1350.png,True,G09998,train,3408de3e280198e91143a159d62d9585,816dc63aea8f78b0
Normal-1351.png,Normal,Normal/images/Normal-1351.png,Normal/masks/Normal-1351.png,True,G09999,train,ece06b31a5a56a20a4fa80b8f655805a,d037c71ae9c938c3
Normal-1352.png,Normal,Normal/images/Normal-1352.png,Normal/masks/Normal-1352.png,True,G10000,train,ad0047d943d0a8dd92aa0f58aac12c7b,957c873aad3d282c
Normal-1353.png,Normal,Normal/images/Normal-1353.png,Normal/masks/Normal-1353.png,True,G10001,train,07b753ad8ef0e1a1ab31bc89ddf572b2,d039c339786939d9
Normal-1354.png,Normal,Normal/images/Normal-1354.png,Normal/masks/Normal-1354.png,True,G10002,train,821b5bea6fb292bff83454d9e70c4637,d96dc20bcd493d68
Normal-1355.png,Normal,Normal/images/Normal-1355.png,Normal/masks/Normal-1355.png,True,G10003,train,4f824f568b4e17691450149d28fa34db,d1fec71aa94c30c6
Normal-1356.png,Normal,Normal/images/Normal-1356.png,Normal/masks/Normal-1356.png,True,G10004,train,ca672968613b1047f04b8942968d1645,956c8f3a8f2d484d
Normal-1357.png,Normal,Normal/images/Normal-1357.png,Normal/masks/Normal-1357.png,True,G10005,train,0547446fd4d60e5ba384abd5e24dc44e,806fc61acd498dcf
Normal-1358.png,Normal,Normal/images/Normal-1358.png,Normal/masks/Normal-1358.png,True,G10006,train,50d53053bfd720ec478ce9209ef2a9ba,d17ae71b99cc1892
Normal-1359.png,Normal,Normal/images/Normal-1359.png,Normal/masks/Normal-1359.png,True,G10007,train,f6d87ac1edbd0126bdc1d7c3d5d37985,d47e873f8d053c84
Normal-136.png,Normal,Normal/images/Normal-136.png,Normal/masks/Normal-136.png,True,G10008,train,8a7ebec2f5e0cd131e335e5945341541,d07cc63a9d2d3cc4
Normal-1360.png,Normal,Normal/images/Normal-1360.png,Normal/masks/Normal-1360.png,True,G10009,train,60e4e8181a0459a529d3f721644bc49a,917f861a8d1d5cc5
Normal-1361.png,Normal,Normal/images/Normal-1361.png,Normal/masks/Normal-1361.png,True,G10010,train,bbffad94910098fe25347492dd86ce21,d16be70ae86969c1
Normal-1362.png,Normal,Normal/images/Normal-1362.png,Normal/masks/Normal-1362.png,True,G10011,val,7865a3ffee6f44dbe12515f4573569cb,d0f6cf9ac98960c6
Normal-1363.png,Normal,Normal/images/Normal-1363.png,Normal/masks/Normal-1363.png,True,G10012,val,d99be2ec74d043566132b532411672ac,902dc65e9eca1d59
Normal-1364.png,Normal,Normal/images/Normal-1364.png,Normal/masks/Normal-1364.png,True,G10013,val,b787fb4da56e6a5a4dd6fee7dcd6bb7e,d4adc239ed3c446c
Normal-1365.png,Normal,Normal/images/Normal-1365.png,Normal/masks/Normal-1365.png,True,G10014,test,81e14c69cd3b6e4d4338e0d2166a9ef0,d82ee34b1e48397c
Normal-1366.png,Normal,Normal/images/Normal-1366.png,Normal/masks/Normal-1366.png,True,G10015,test,eed62cf13b9aa771e472ed3659581ee7,81a5c60baeca79d6
Normal-1367.png,Normal,Normal/images/Normal-1367.png,Normal/masks/Normal-1367.png,True,G10016,test,b05807169c875a80b320c0ebdd435ac5,d47e871f8d0c1cc6
Normal-1368.png,Normal,Normal/images/Normal-1368.png,Normal/masks/Normal-1368.png,True,G10017,train,39f81041b25d4aeef61b8fc44b8fd45f,911ed64a1e861db7
Normal-1369.png,Normal,Normal/images/Normal-1369.png,Normal/masks/Normal-1369.png,True,G10018,train,bd2be0b3212bb9b4d6852d27935140ca,d828c23bcb7d784c
Normal-137.png,Normal,Normal/images/Normal-137.png,Normal/masks/Normal-137.png,True,G10019,train,3038651a0211bcba24b110d275f14541,997bd65a181919d3
Normal-1370.png,Normal,Normal/images/Normal-1370.png,Normal/masks/Normal-1370.png,True,G10020,train,de6dbd8311897b21f4ed6b147d09f094,913fc2389ce13d96
Normal-1371.png,Normal,Normal/images/Normal-1371.png,Normal/masks/Normal-1371.png,True,G10021,train,15c7dae9a596238a76efe110824f9f27,812ec61acc799dcb
Normal-1372.png,Normal,Normal/images/Normal-1372.png,Normal/masks/Normal-1372.png,True,G10022,train,2a2209071d5482aa0ed0043016f569f5,d47e821bed852ce4
Normal-1373.png,Normal,Normal/images/Normal-1373.png,Normal/masks/Normal-1373.png,True,G10023,train,fbfe63b7e20b4fbb67e8c836fa46054b,d02dc23beb6f6860
Normal-1374.png,Normal,Normal/images/Normal-1374.png,Normal/masks/Normal-1374.png,True,G10024,train,1e823161980ac058f580b7a5d7e74db4,996fc65ac54b184d
Normal-1375.png,Normal,Normal/images/Normal-1375.png,Normal/masks/Normal-1375.png,True,G10025,train,598195d6e8589e4e13a4a38142fb35fb,913f8e1a9c4b3dc2
Normal-1376.png,Normal,Normal/images/Normal-1376.png,Normal/masks/Normal-1376.png,True,G10026,train,454d9c04a77aae14c430ee166455b49f,987ad25adc5c98c6
Normal-1377.png,Normal,Normal/images/Normal-1377.png,Normal/masks/Normal-1377.png,True,G10027,train,ed87f92dd714f96f4d3a3707a5e3ab2b,907bc71ad9591cd1
Normal-1378.png,Normal,Normal/images/Normal-1378.png,Normal/masks/Normal-1378.png,True,G10028,train,6f939db5cabc79305faa97e174a674e1,d42fc71b8dc91994
Normal-1379.png,Normal,Normal/images/Normal-1379.png,Normal/masks/Normal-1379.png,True,G10029,train,024e73b72bd930aaa4a1825e97487711,c06fcf1acd492ac3
Normal-138.png,Normal,Normal/images/Normal-138.png,Normal/masks/Normal-138.png,True,G10030,val,2baac47918cf9135172c1b4f9f90dcab,d13dc31998c99ada
Normal-1380.png,Normal,Normal/images/Normal-1380.png,Normal/masks/Normal-1380.png,True,G10031,train,f201d6663a462848a3ae53a6a307ea21,d46ec75e8d4d28c4
Normal-1381.png,Normal,Normal/images/Normal-1381.png,Normal/masks/Normal-1381.png,True,G10032,val,89fdb2875056739a4a1bdf252d9016e7,957e871e0d0d1ccd
Normal-1382.png,Normal,Normal/images/Normal-1382.png,Normal/masks/Normal-1382.png,True,G10033,test,dec3665c770cb0d61a8636949756a5df,906de63aa94d3ccc
Normal-1383.png,Normal,Normal/images/Normal-1383.png,Normal/masks/Normal-1383.png,True,G10034,train,1b7ff19f5a1a95d8609545fcac505e81,912cc63afd6938e4
Normal-1384.png,Normal,Normal/images/Normal-1384.png,Normal/masks/Normal-1384.png,True,G10035,train,bcab69496fec98f86981d67cb8b28fe3,917dce3a891939cc
Normal-1385.png,Normal,Normal/images/Normal-1385.png,Normal/masks/Normal-1385.png,True,G10036,test,eaada832cfb560e41ca41c6f869cfe6c,d07cc63abd2d3c84
Normal-1386.png,Normal,Normal/images/Normal-1386.png,Normal/masks/Normal-1386.png,True,G10037,train,47919887ed0ad65ac50b7172463a636d,95fc871aad2d6834
Normal-1387.png,Normal,Normal/images/Normal-1387.png,Normal/masks/Normal-1387.png,True,G10038,val,c9f5014ecbe00df249aa643828a83c72,91b5c71bbca42ba4
Normal-1388.png,Normal,Normal/images/Normal-1388.png,Normal/masks/Normal-1388.png,True,G10039,train,f342c1cae6bbca86b8f38177e9b95ca0,91f8871aba3c392d
Normal-1389.png,Normal,Normal/images/Normal-1389.png,Normal/masks/Normal-1389.png,True,G10040,test,3597adb8560b4c422b186fd890d56403,d44fe71fad8d0c80
Normal-139.png,Normal,Normal/images/Normal-139.png,Normal/masks/Normal-139.png,True,G10041,train,602865050a08d82362ad72634708db28,d02bca5a5cd81e9e
Normal-1390.png,Normal,Normal/images/Normal-1390.png,Normal/masks/Normal-1390.png,True,G10042,train,bfa2bc10097f0bdafaba1b662bbf39fe,c56eef1ac9871b40
Normal-1391.png,Normal,Normal/images/Normal-1391.png,Normal/masks/Normal-1391.png,True,G10043,val,716a70e132e3295606780f9afa461161,d12b7e4a7c40693d
Normal-1392.png,Normal,Normal/images/Normal-1392.png,Normal/masks/Normal-1392.png,True,G10044,train,b9d2b94eae57517a6381d4c0aaac5148,d03dea3d3c49493c
Normal-1393.png,Normal,Normal/images/Normal-1393.png,Normal/masks/Normal-1393.png,True,G10045,train,8624e9011bfa44d1bea5ebdc0a91209c,d53a525b1c481f97
Normal-1394.png,Normal,Normal/images/Normal-1394.png,Normal/masks/Normal-1394.png,True,G10046,train,b68fcbbbc0cc1d1b46f719c7dfbfc0f6,d121e70bbc6939f0
Normal-1395.png,Normal,Normal/images/Normal-1395.png,Normal/masks/Normal-1395.png,True,G10047,train,7d8c60bb8c6adeaa591ecdaccd3dde08,c097d30db460b7e9
Normal-1396.png,Normal,Normal/images/Normal-1396.png,Normal/masks/Normal-1396.png,True,G10048,train,76f00a6d9b3f14ff70b743dd59b27c6c,d13bc25a3879399a
Normal-1397.png,Normal,Normal/images/Normal-1397.png,Normal/masks/Normal-1397.png,True,G10049,train,91d1038c3342d052090108ea1b3c69ee,d46fa70d25246ce6
Normal-1398.png,Normal,Normal/images/Normal-1398.png,Normal/masks/Normal-1398.png,True,G10050,train,6089f00af99aeff335a59a2f1b5f953c,912bc24a5cd859df
Normal-1399.png,Normal,Normal/images/Normal-1399.png,Normal/masks/Normal-1399.png,True,G10051,test,fe4557741d2da6d92d84e083855bc64c,916cc71a99ed38e4
Normal-14.png,Normal,Normal/images/Normal-14.png,Normal/masks/Normal-14.png,True,G10052,val,00b53ecba8ec61de19096b38f526bec7,d979c65a5c405b59
Normal-140.png,Normal,Normal/images/Normal-140.png,Normal/masks/Normal-140.png,True,G10053,test,248b1a7bd7bd5b1477539a63f92514f5,907dcf38994918cf
Normal-1400.png,Normal,Normal/images/Normal-1400.png,Normal/masks/Normal-1400.png,True,G10054,train,b59eb73612ca91a16a0db7f7115beb71,d47ec34fbc682c90
Normal-1401.png,Normal,Normal/images/Normal-1401.png,Normal/masks/Normal-1401.png,True,G10055,val,4a68c327bc27f9ab6889f1da8883419b,957ee71bb8c909c0
Normal-1402.png,Normal,Normal/images/Normal-1402.png,Normal/masks/Normal-1402.png,True,G10056,test,8901d328baa13a63f0236d96c7edd7d1,d46e870b8d555ac6
Normal-1403.png,Normal,Normal/images/Normal-1403.png,Normal/masks/Normal-1403.png,True,G10057,train,c295c127eb955c0d97fbca6007845201,c93fe61a7c113919
Normal-1404.png,Normal,Normal/images/Normal-1404.png,Normal/masks/Normal-1404.png,True,G10058,train,8bdaf6da73f6f34530e731b4a98a118d,d42d6bc34bc92fc0
Normal-1405.png,Normal,Normal/images/Normal-1405.png,Normal/masks/Normal-1405.png,True,G10059,train,60da7274d915c7464404e12adad32c30,986dc55e454f1e49
Normal-1406.png,Normal,Normal/images/Normal-1406.png,Normal/masks/Normal-1406.png,True,G10060,train,6fc18e58c18a094f5262a319a2744cb3,dd28521b1f0c3f1d
Normal-1407.png,Normal,Normal/images/Normal-1407.png,Normal/masks/Normal-1407.png,True,G10061,train,1abe053cb9a01dc83e0f17e857ae2a81,d939421c4c5e3d6d
Normal-1408.png,Normal,Normal/images/Normal-1408.png,Normal/masks/Normal-1408.png,True,G10062,train,3deb61a6f6542d1005de9dbe27903cb5,8919161e1e7a787e
Normal-1409.png,Normal,Normal/images/Normal-1409.png,Normal/masks/Normal-1409.png,True,G10063,train,437e0475a3550d7c860ad9dbe2951e54,d92dc21b594d3a59
Normal-141.png,Normal,Normal/images/Normal-141.png,Normal/masks/Normal-141.png,True,G10064,test,948b0ada6c5a0cf002a246e707fa027d,917ace5a58597951
Normal-1410.png,Normal,Normal/images/Normal-1410.png,Normal/masks/Normal-1410.png,True,G10065,train,9ed147d49c875d5dc2e47159fd08c289,896f865b444eb8a7
Normal-1411.png,Normal,Normal/images/Normal-1411.png,Normal/masks/Normal-1411.png,True,G10066,train,39f4b5a41b0805b22de1bbd85e6eb94c,d97dc61b1c08391f
Normal-1412.png,Normal,Normal/images/Normal-1412.png,Normal/masks/Normal-1412.png,True,G10067,train,f61a33d2fe37835cbdaf054b1e941c0c,dd2e50530b4f4f0e
Normal-1413.png,Normal,Normal/images/Normal-1413.png,Normal/masks/Normal-1413.png,True,G10068,train,e1c44b96679658be019fbdb9a72a89fb,997b07161e555e1a
Normal-1414.png,Normal,Normal/images/Normal-1414.png,Normal/masks/Normal-1414.png,True,G10069,train,18c684e7ad1872971252f50b1b3f0a5e,954e0e4b4d6b4763
Normal-1415.png,Normal,Normal/images/Normal-1415.png,Normal/masks/Normal-1415.png,True,G10070,train,f3cc36c940b0f44abf383d5e4b4d700f,fd7dd4c2430f4d40
Normal-1416.png,Normal,Normal/images/Normal-1416.png,Normal/masks/Normal-1416.png,True,G10071,train,cc082bdbc1b9982fb6f63949ff8451ec,996c071a5b4d7a78
Normal-1417.png,Normal,Normal/images/Normal-1417.png,Normal/masks/Normal-1417.png,True,G10072,val,2bb71485d49f3dda2a324511a27d268b,993f861a974e18b5
Normal-1418.png,Normal,Normal/images/Normal-1418.png,Normal/masks/Normal-1418.png,True,G10073,val,366007488d2172094154b9dcf3b58ebb,d929c69e544a1d5b
Normal-1419.png,Normal,Normal/images/Normal-1419.png,Normal/masks/Normal-1419.png,True,G10074,val,6930276a6bfcd742b496e0a434e80230,d96906195f6b39a4
Normal-142.png,Normal,Normal/images/Normal-142.png,Normal/masks/Normal-142.png,True,G10075,test,d01526de5fb16827f464c8df2de4ce34,d47e873f0d0d09c3
Normal-1420.png,Normal,Normal/images/Normal-1420.png,Normal/masks/Normal-1420.png,True,G10076,test,0df27646907f68ac219e424f23d11d98,9c4dc61b0d4f1b4b
Normal-1421.png,Normal,Normal/images/Normal-1421.png,Normal/masks/Normal-1421.png,True,G10077,train,d2086709b86fcae6b4f02f52116ea7aa,897a57061a975e1e
Normal-1422.png,Normal,Normal/images/Normal-1422.png,Normal/masks/Normal-1422.png,True,G10078,train,a96ac071855da18a627d09723ef172e4,cc3cd2431f431a5f
Normal-1423.png,Normal,Normal/images/Normal-1423.png,Normal/masks/Normal-1423.png,True,G10079,train,27d4ccb672bebc1f31856977817bbb9d,9928061b5d4d79f5
Normal-1424.png,Normal,Normal/images/Normal-1424.png,Normal/masks/Normal-1424.png,True,G10080,train,b36a24548fcb568a4805c65cbd20dc4f,897e570316c71e99
Normal-1425.png,Normal,Normal/images/Normal-1425.png,Normal/masks/Normal-1425.png,True,G10081,train,0b9e9718ac8ff2f63002727c49c167bf,d92d071ac71a554f
Normal-1426.png,Normal,Normal/images/Normal-1426.png,Normal/masks/Normal-1426.png,True,G10082,train,6ba634f478e10691a6c0da5c300dde25,997f871a055a19cd
Normal-1427.png,Normal,Normal/images/Normal-1427.png,Normal/masks/Normal-1427.png,True,G10083,train,35d55fe5c384c08cb443c6e120132e31,dc3dc3439f81a30b
Normal-1428.png,Normal,Normal/images/Normal-1428.png,Normal/masks/Normal-1428.png,True,G10084,train,a5bec2c7ba695644590ae532ba90e45f,897b46165ec65b98
Normal-1429.png,Normal,Normal/images/Normal-1429.png,Normal/masks/Normal-1429.png,True,G10085,train,a0864653495cb00ed543d33dfa91c55e,c02846d71bd5475f
Normal-143.png,Normal,Normal/images/Normal-143.png,Normal/masks/Normal-143.png,True,G10086,train,c769c87618a97731cc91eee77b620105,c56f0f4949ea78e0
Normal-1430.png,Normal,Normal/images/Normal-1430.png,Normal/masks/Normal-1430.png,True,G10087,train,8448448b637c45e69ff5289e5632aa50,d42f861f1f033d1c
Normal-1431.png,Normal,Normal/images/Normal-1431.png,Normal/masks/Normal-1431.png,True,G10088,train,3313eaa37511936fe35d9f51b6e27c6a,dc2a435d431f454f
Normal-1432.png,Normal,Normal/images/Normal-1432.png,Normal/masks/Normal-1432.png,True,G10089,train,69e840c7234daaf6bf8b094562243571,cc3fc30b5d4b1c4c
Normal-1433.png,Normal,Normal/images/Normal-1433.png,Normal/masks/Normal-1433.png,True,G10090,train,f57312bf513bd284169e1de84933009f,bc6dc31b0d5d0c4d
Normal-1434.png,Normal,Normal/images/Normal-1434.png,Normal/masks/Normal-1434.png,True,G10091,val,be0d2dc16ae27660ada20a4293083472,b939071e1c59595d
Normal-1435.png,Normal,Normal/images/Normal-1435.png,Normal/masks/Normal-1435.png,True,G10092,val,d532c95b7662d73ae375a2264ebb1bfd,d93dc61e154b1d49
Normal-1436.png,Normal,Normal/images/Normal-1436.png,Normal/masks/Normal-1436.png,True,G10093,val,46d746879a47ad0bfee568bd07ba5677,cc3d421f094f3dac
Normal-1437.png,Normal,Normal/images/Normal-1437.png,Normal/masks/Normal-1437.png,True,G10094,test,4ad907f17279847bfb8de04b4b2c9abe,993dc45ad54f1e44
Normal-1438.png,Normal,Normal/images/Normal-1438.png,Normal/masks/Normal-1438.png,True,G10095,test,d2b7d1948b838bed0935e408c071fe0f,997d57021f45293b
Normal-1439.png,Normal,Normal/images/Normal-1439.png,Normal/masks/Normal-1439.png,True,G10096,test,2191c32659f80fb644032fa9a1eff732,d43dc34b4d4a497c
Normal-144.png,Normal,Normal/images/Normal-144.png,Normal/masks/Normal-144.png,True,G10097,train,d4cd26b8f4b8a9fa0ede4c0b7185a521,916a4e5a4d4d4dc7
Normal-1440.png,Normal,Normal/images/Normal-1440.png,Normal/masks/Normal-1440.png,True,G10098,train,5037fb389f0ce881be7ef4870b1df99a,dd68c2964fc21b4b
Normal-1441.png,Normal,Normal/images/Normal-1441.png,Normal/masks/Normal-1441.png,True,G10099,train,0bbf2e54bbdc14025a723d95c2a0eb96,897b87961cc55b19
Normal-1442.png,Normal,Normal/images/Normal-1442.png,Normal/masks/Normal-1442.png,True,G10100,train,16fa12bf04e7e22752389d1cf58fd6d3,997f07061a096f3e
Normal-1443.png,Normal,Normal/images/Normal-1443.png,Normal/masks/Normal-1443.png,True,G10101,train,31b98111a1b4d970645e39aeca1b2e7d,cd3cd24f450b3d0b
Normal-1444.png,Normal,Normal/images/Normal-1444.png,Normal/masks/Normal-1444.png,True,G10102,train,62f2f433d01e6c3b59d743cd94028b21,897f06161e47471f
Normal-1445.png,Normal,Normal/images/Normal-1445.png,Normal/masks/Normal-1445.png,True,G10103,train,3e1554cff527ba8f08f19690b8eeec7f,997b851a5b0e38e9
Normal-1446.png,Normal,Normal/images/Normal-1446.png,Normal/masks/Normal-1446.png,True,G10104,train,9fe292e3d4ba31c621bdf2b4cc66c0ea,817906165a5fda7a
Normal-1447.png,Normal,Normal/images/Normal-1447.png,Normal/masks/Normal-1447.png,True,G10105,train,abff3d1e7ea6ccb3f3a1ec68556bfce5,bd3cd31b55521d44
Normal-1448.png,Normal,Normal/images/Normal-1448.png,Normal/masks/Normal-1448.png,True,G10106,train,0c0423f710553395d35ff9dee996d26c,9178c7061a5d4f3e
Normal-1449.png,Normal,Normal/images/Normal-1449.png,Normal/masks/Normal-1449.png,True,G10107,train,7076a7accb723b15acbe2685c4762022,dd3d42421f934b0f
Normal-145.png,Normal,Normal/images/Normal-145.png,Normal/masks/Normal-145.png,True,G10108,train,f4c3ee27b2dc8114f178ab549b1506b7,c06f4758e1e4e8e3
Normal-1450.png,Normal,Normal/images/Normal-1450.png,Normal/masks/Normal-1450.png,True,G10109,train,41ff015993b134af22ead668c33a64dc,d46f430b4e2d3d07
Normal-1451.png,Normal,Normal/images/Normal-1451.png,Normal/masks/Normal-1451.png,True,G10110,val,039264ea476ab86feef730b3728fcd89,8979061e1f4d5b59
Normal-1452.png,Normal,Normal/images/Normal-1452.png,Normal/masks/Normal-1452.png,True,G10111,train,441d61e043abaeef8ffa0b8445089f8a,992946071f59697e
Normal-1453.png,Normal,Normal/images/Normal-1453.png,Normal/masks/Normal-1453.png,True,G10112,val,e8aeade66ab4d38b630d29cc9b262591,d03db2030f1d5f4b
Normal-1454.png,Normal,Normal/images/Normal-1454.png,Normal/masks/Normal-1454.png,True,G10113,val,d3365c219f023d6f749a4a8dece30827,913a468a9f85eeb8
Normal-1455.png,Normal,Normal/images/Normal-1455.png,Normal/masks/Normal-1455.png,True,G10114,test,b8c68466df234e12469581a254142ced,d42d874b6c707873
Normal-1456.png,Normal,Normal/images/Normal-1456.png,Normal/masks/Normal-1456.png,True,G10115,test,6bdd324838cb90503cdc5fdd223bb748,cd5c534b1f0a0fa6
Normal-1457.png,Normal,Normal/images/Normal-1457.png,Normal/masks/Normal-1457.png,True,G10116,test,e59d1b73e811cb0f9af33c4f39ea8f8a,816b06161fde3a97
Normal-1458.png,Normal,Normal/images/Normal-1458.png,Normal/masks/Normal-1458.png,True,G10117,train,c1a9335286abeaf6ed8612fdb32c37c6,987f85d61e0d591c
Normal-1459.png,Normal,Normal/images/Normal-1459.png,Normal/masks/Normal-1459.png,True,G10118,train,d59645867c2f650f81c1877113c5957e,987f47465e494d1c
Normal-146.png,Normal,Normal/images/Normal-146.png,Normal/masks/Normal-146.png,True,G10119,train,3d8ec2d8e09102c0259e50846299d024,8129ea78785b7955
Normal-1460.png,Normal,Normal/images/Normal-1460.png,Normal/masks/Normal-1460.png,True,G10120,train,0b0a26eeb96c30d60bb5724f73b9149e,91609e1a5a5f7a1b
Normal-1461.png,Normal,Normal/images/Normal-1461.png,Normal/masks/Normal-1461.png,True,G10121,train,2a562a7e96b76cca54bab21d99059e37,c879d7061e411f1f
Normal-1462.png,Normal,Normal/images/Normal-1462.png,Normal/masks/Normal-1462.png,True,G10122,train,ccf3671e8a79b91ef7de27e22bd8d7af,dc3cd34a5f454859
Normal-1463.png,Normal,Normal/images/Normal-1463.png,Normal/masks/Normal-1463.png,True,G10123,train,cdd7030c97e4a1dcd24b67e0e85061a5,fc5cc2535e4a4a1e
Normal-1464.png,Normal,Normal/images/Normal-1464.png,Normal/masks/Normal-1464.png,True,G10124,train,49eebd52ef998e07e54586413ce0e822,dd2ec24b094e2f3c
Normal-1465.png,Normal,Normal/images/Normal-1465.png,Normal/masks/Normal-1465.png,True,G10125,train,7258cdfaab495e406238a896e982bf53,81799e06fa1b19ec
Normal-1466.png,Normal,Normal/images/Normal-1466.png,Normal/masks/Normal-1466.png,True,G10126,train,cb45c3f7cc484b7e2cb3cbca4ea11faa,85684e686dc6e7c3
Normal-1467.png,Normal,Normal/images/Normal-1467.png,Normal/masks/Normal-1467.png,True,G10127,train,82bc8f6a2a7ef5300248c5bb25e72abc,883d071f1e071f1f
Normal-1468.png,Normal,Normal/images/Normal-1468.png,Normal/masks/Normal-1468.png,True,G10128,train,2ce0472bc36913f87af541b52c2b985a,8928861a938fbbb7
Normal-1469.png,Normal,Normal/images/Normal-1469.png,Normal/masks/Normal-1469.png,True,G10129,val,9876377e0b2f65302071309557bded0a,d42c42df494cee2d
Normal-147.png,Normal,Normal/images/Normal-147.png,Normal/masks/Normal-147.png,True,G10130,val,ace5335664545c506ded32adfb617614,d821c61efe783d60
Normal-1470.png,Normal,Normal/images/Normal-1470.png,Normal/masks/Normal-1470.png,True,G10131,train,e1e8303d294c2f7ff52e384f3da27273,dc3cd25b0dd40a2f
Normal-1471.png,Normal,Normal/images/Normal-1471.png,Normal/masks/Normal-1471.png,True,G10132,test,3806b760ec51435deeb0aad0a1604143,997906065add6a79
Normal-1472.png,Normal,Normal/images/Normal-1472.png,Normal/masks/Normal-1472.png,True,G10133,train,ef594e82c3a19ab9e7bee987b5385351,9969871e1a4d1f59
Normal-1473.png,Normal,Normal/images/Normal-1473.png,Normal/masks/Normal-1473.png,True,G10134,test,94eb6d3fc2834cf50676ccec411a3f1b,812956963ad96fc5
Normal-1474.png,Normal,Normal/images/Normal-1474.png,Normal/masks/Normal-1474.png,True,G10135,train,d1f7122daf85b3bc7be68b8300ca2ed4,917886c61b5dce1e
Normal-1475.png,Normal,Normal/images/Normal-1475.png,Normal/masks/Normal-1475.png,True,G10136,val,1e8cf6805140c873b3092baee1f9c561,d03f021f184e797d
Normal-1476.png,Normal,Normal/images/Normal-1476.png,Normal/masks/Normal-1476.png,True,G10137,train,33b7a2c7ea5759549352d855530d2ba2,d939461f186e3be0
Normal-1477.png,Normal,Normal/images/Normal-1477.png,Normal/masks/Normal-1477.png,True,G10138,val,891c55a4af25f7101115ca54c532123a,dc3dc3480a6a3d2f
Normal-1478.png,Normal,Normal/images/Normal-1478.png,Normal/masks/Normal-1478.png,True,G10139,test,2a49d3620daaffb3f48b930e9524d543,cd2ec41b91c61fb8
Normal-1479.png,Normal,Normal/images/Normal-1479.png,Normal/masks/Normal-1479.png,True,G10140,train,bfbb33c4732afaf0db2c176b11b84dd1,d43d4b0b0f0a7d1d
Normal-148.png,Normal,Normal/images/Normal-148.png,Normal/masks/Normal-148.png,True,G10141,train,7954a375a3ddd8f4f8512fd14aa84e02,917ce73aab1d282c
Normal-1480.png,Normal,Normal/images/Normal-1480.png,Normal/masks/Normal-1480.png,True,G10142,train,a1a7ec1b5e09cd667821df77283be29b,991a861b587e7ae8
Normal-1481.png,Normal,Normal/images/Normal-1481.png,Normal/masks/Normal-1481.png,True,G10143,train,3cc8b808436ead0b0f1d64d9f6fcd23b,cc3f520b464f1e4b
Normal-1482.png,Normal,Normal/images/Normal-1482.png,Normal/masks/Normal-1482.png,True,G10144,train,8b8be82184bc0307afab39afc095628d,8979d61e18867a79
Normal-1483.png,Normal,Normal/images/Normal-1483.png,Normal/masks/Normal-1483.png,True,G10145,train,e6535a8b340cacc3a33898559af81e2d,d42d4a531f0c4e6f
Normal-1484.png,Normal,Normal/images/Normal-1484.png,Normal/masks/Normal-1484.png,True,G10146,train,76b64a3674ba5c9de44ec2758506ed03,8979871a1e4d3b1b
Normal-1485.png,Normal,Normal/images/Normal-1485.png,Normal/masks/Normal-1485.png,True,G10147,train,b1dc2050c26d7fdde511951248b123b6,dd2e52430d6c2c3f
Normal-1486.png,Normal,Normal/images/Normal-1486.png,Normal/masks/Normal-1486.png,True,G10148,test,645b4c14a50e2550b3b0c74d9e2b2840,8028c75e1edd5b1d
Normal-1487.png,Normal,Normal/images/Normal-1487.png,Normal/masks/Normal-1487.png,True,G10149,train,2311ea6c96b7c4ba5246794ebebdb3c7,d43d4b0b4b0f4f0d
Normal-1488.png,Normal,Normal/images/Normal-1488.png,Normal/masks/Normal-1488.png,True,G10150,train,3a679e7362a76b6399595cada9b6da87,c87c52435b1dac3d
Normal-1489.png,Normal,Normal/images/Normal-1489.png,Normal/masks/Normal-1489.png,True,G10151,train,7c42a0c4e8157ef88d61f374bc8fe553,9979061a394d3be5
Normal-149.png,Normal,Normal/images/Normal-149.png,Normal/masks/Normal-149.png,True,G10152,train,4543549880a9de13ec71699579828768,d12dc71ecc4d1cc5
Normal-1490.png,Normal,Normal/images/Normal-1490.png,Normal/masks/Normal-1490.png,True,G10153,train,7487d323f6bd758918f0e6a5ff8de030,dc2d42831b1bed4e
Normal-1491.png,Normal,Normal/images/Normal-1491.png,Normal/masks/Normal-1491.png,True,G10154,val,44ff40228dd79dc6dae3b22f45b3fbe1,d97905861e1e5f1c
Normal-1492.png,Normal,Normal/images/Normal-1492.png,Normal/masks/Normal-1492.png,True,G10155,test,c478f4a98e7617983080043ad37004f5,c9fbc51ac5571a48
Normal-1493.png,Normal,Normal/images/Normal-1493.png,Normal/masks/Normal-1493.png,True,G10156,train,222c705130880c9926fb1db1f8e3e092,95694e161b1eaf38
Normal-1494.png,Normal,Normal/images/Normal-1494.png,Normal/masks/Normal-1494.png,True,G10157,train,f1f3ed458def02f992daec7163a48a84,cd2c42551b46bdf8
Normal-1495.png,Normal,Normal/images/Normal-1495.png,Normal/masks/Normal-1495.png,True,G10158,train,8ef94bdb26bcd4377951ab448f06d2c4,903b56563cc16be5
Normal-1496.png,Normal,Normal/images/Normal-1496.png,Normal/masks/Normal-1496.png,True,G10159,train,0d72310afc5d753813223768178e3ec4,d13b865a190cfb39
Normal-1497.png,Normal,Normal/images/Normal-1497.png,Normal/masks/Normal-1497.png,True,G10160,train,5f95fba7b72fd6238b1f7811d8361422,987984161b8ddf3a
Normal-1498.png,Normal,Normal/images/Normal-1498.png,Normal/masks/Normal-1498.png,True,G10161,val,d5244feb04d08020245c52e34348ddc1,997d461a196d3bb0
Normal-1499.png,Normal,Normal/images/Normal-1499.png,Normal/masks/Normal-1499.png,True,G10162,val,27f2c40579623e90270a480a93503013,d52dc2d75f404c1b
Normal-15.png,Normal,Normal/images/Normal-15.png,Normal/masks/Normal-15.png,True,G10163,test,2e4aa79b094e5aec5c43512cdccf399c,993d865a0f4d5a5c
Normal-150.png,Normal,Normal/images/Normal-150.png,Normal/masks/Normal-150.png,True,G10164,test,74b489ffc423b6f5d602054f15af7268,913dc63ad84939d9
Normal-1500.png,Normal,Normal/images/Normal-1500.png,Normal/masks/Normal-1500.png,True,G10165,train,458a9b4a36e2a44e2c6fc3375be9e05a,c02d0f47c9ceec2d
Normal-1501.png,Normal,Normal/images/Normal-1501.png,Normal/masks/Normal-1501.png,True,G10166,train,385fa89c899909d5cdcedfbc117c9d57,d939d21f554a3949
Normal-1502.png,Normal,Normal/images/Normal-1502.png,Normal/masks/Normal-1502.png,True,G10167,train,c6104db3ab7548e101747cc623a82b94,9879c55650dd1ad9
Normal-1503.png,Normal,Normal/images/Normal-1503.png,Normal/masks/Normal-1503.png,True,G10168,train,0ed8f0dc641df913369d07085875ce79,9168861e5aed7a69
Normal-1504.png,Normal,Normal/images/Normal-1504.png,Normal/masks/Normal-1504.png,True,G10169,train,52dd408c4e3939f3e6212548b4704159,d93b46c65a086f1b
Normal-1505.png,Normal,Normal/images/Normal-1505.png,Normal/masks/Normal-1505.png,True,G10170,train,59c0bbd30c1300bb6e6efa13f124c94b,fc2ec3d3094f0b0b
Normal-1506.png,Normal,Normal/images/Normal-1506.png,Normal/masks/Normal-1506.png,True,G10171,train,5b2b9c9f8ce777e5a44020e8a457dc37,dd3cd24b4b420f1d
Normal-1507.png,Normal,Normal/images/Normal-1507.png,Normal/masks/Normal-1507.png,True,G10172,train,886f274ada542af77524d805887ee3e6,996cc01bc50b9ecf
Normal-1508.png,Normal,Normal/images/Normal-1508.png,Normal/masks/Normal-1508.png,True,G10173,train,7572c1b083104de5ece1d40b5a856aba,992d861a1f4f1b4d
Normal-1509.png,Normal,Normal/images/Normal-1509.png,Normal/masks/Normal-1509.png,True,G10174,val,9af34f41429e779872b5c9f28ad2c81d,882f07961a5f5d5a
Normal-151.png,Normal,Normal/images/Normal-151.png,Normal/masks/Normal-151.png,True,G10175,val,617689b9eab722e6b716e1c5139e1615,913dc3399cc939b1
Normal-1510.png,Normal,Normal/images/Normal-1510.png,Normal/masks/Normal-1510.png,True,G10176,test,4836901841cd300b91207e4d2fc106f5,d95d4716194d0f1e
Normal-1511.png,Normal,Normal/images/Normal-1511.png,Normal/masks/Normal-1511.png,True,G10177,test,bfe16b3dc48c46f1c76a3c3e96ec9a93,cc2ed20f0de0aab7
Normal-1512.png,Normal,Normal/images/Normal-1512.png,Normal/masks/Normal-1512.png,True,G10178,train,5fdeb5910c44b48c3c6b6b1636727508,997b05161ac55eb9
Normal-1513.png,Normal,Normal/images/Normal-1513.png,Normal/masks/Normal-1513.png,True,G10179,train,2e1bf6305ad2830078540475a4f2da46,9168161f191f1b5f
Normal-1514.png,Normal,Normal/images/Normal-1514.png,Normal/masks/Normal-1514.png,True,G10180,train,bb795e43fd61e75a27da5aa545a947b2,dc2ec25b0d6c2e78
Normal-1515.png,Normal,Normal/images/Normal-1515.png,Normal/masks/Normal-1515.png,True,G10181,train,8f953fb1d3d638532ef62b43355f3191,dd3cc20b4c0b3dec
Normal-1516.png,Normal,Normal/images/Normal-1516.png,Normal/masks/Normal-1516.png,True,G10182,train,af15be6e61bae9a456ab1f2615c5ec89,9c1c431b1f4f0e5e
Normal-1517.png,Normal,Normal/images/Normal-1517.png,Normal/masks/Normal-1517.png,True,G10183,train,0b6bc8bfd9b5ddd5e328f60d0b4b1dbb,c86ed2571f470b19
Normal-1518.png,Normal,Normal/images/Normal-1518.png,Normal/masks/Normal-1518.png,True,G10184,train,2070278261ac7daac3c4367cd77276c5,997f051e174f1a4c
Normal-1519.png,Normal,Normal/images/Normal-1519.png,Normal/masks/Normal-1519.png,True,G10185,train,4440173a08d9b0aade7276c47f72e595,817f96161a5d561b
Normal-152.png,Normal,Normal/images/Normal-152.png,Normal/masks/Normal-152.png,True,G10186,train,62e736a3d03ef52a1bd62684c1a3dad8,d07ec71a992d3c86
Normal-1520.png,Normal,Normal/images/Normal-1520.png,Normal/masks/Normal-1520.png,True,G10187,train,3343f895d523f5cd1707aa1e78196603,dc3dc34b4a4a6f0a
Normal-1521.png,Normal,Normal/images/Normal-1521.png,Normal/masks/Normal-1521.png,True,G10188,val,ca3d9c7473250e30afc9fd290f5cd822,817a571e18975a9b
Normal-1522.png,Normal,Normal/images/Normal-1522.png,Normal/masks/Normal-1522.png,True,G10189,train,99878ce7673bb282c9e30fc785de8590,817dc7821b454f1f
Normal-1523.png,Normal,Normal/images/Normal-1523.png,Normal/masks/Normal-1523.png,True,G10190,train,b6b22d6addcc3d609f90177fdde2a1ba,917857469a5bc54e
Normal-1524.png,Normal,Normal/images/Normal-1524.png,Normal/masks/Normal-1524.png,True,G10191,train,219ca6c51be09a5edbf32d7fa80b3d7e,d929c21796c71e1b
Normal-1525.png,Normal,Normal/images/Normal-1525.png,Normal/masks/Normal-1525.png,True,G10192,val,1cce7d3d25a6096f2c239ab20b159194,c43ee1cb8fc1901f
Normal-1526.png,Normal,Normal/images/Normal-1526.png,Normal/masks/Normal-1526.png,True,G10193,train,2743b805869a4a7dc158a98eb443ad6a,997907841e5b691f
Normal-1527.png,Normal,Normal/images/Normal-1527.png,Normal/masks/Normal-1527.png,True,G10194,val,82c673bbca6c7fcbc304f84f4a2a8f52,dc2ed24b1f444e1e
Normal-1528.png,Normal,Normal/images/Normal-1528.png,Normal/masks/Normal-1528.png,True,G10195,test,f69ca3219db8dfde706744c002f446e5,d43d02d25d4b564f
Normal-1529.png,Normal,Normal/images/Normal-1529.png,Normal/masks/Normal-1529.png,True,G10196,test,1117414d8268c93738888d4de8c11f34,993b9618516979c7
Normal-153.png,Normal,Normal/images/Normal-153.png,Normal/masks/Normal-153.png,True,G10197,test,282703e08342637daab8378565d9051a,d46fef1f0d0c0cc4
Normal-1530.png,Normal,Normal/images/Normal-1530.png,Normal/masks/Normal-1530.png,True,G10198,val,1acb786f5a5c9bbc4cc007d860a95fe5,d12b52071a1e6f5b
Normal-1531.png,Normal,Normal/images/Normal-1531.png,Normal/masks/Normal-1531.png,True,G10199,train,1ee3b6eb0030d3fb2ddacbf00a5a0c67,c83d421f0b3f3c2d
Normal-1532.png,Normal,Normal/images/Normal-1532.png,Normal/masks/Normal-1532.png,True,G10200,train,f4feb710053e0f3a7f9d263248f1fa75,c42d444f0e5eae7c
Normal-1533.png,Normal,Normal/images/Normal-1533.png,Normal/masks/Normal-1533.png,True,G10201,train,7d81c8b8247227c16909af59a1c02d06,c03f034d1a6e6dba
Normal-1534.png,Normal,Normal/images/Normal-1534.png,Normal/masks/Normal-1534.png,True,G10202,train,5a3bc53e4bf145d661942e0ec2bdece8,dc3cd25b59c50f18
Normal-1535.png,Normal,Normal/images/Normal-1535.png,Normal/masks/Normal-1535.png,True,G10203,train,3ad53457fc89c9ef52a1789d0d1e9256,dc3d42165b4f1e49
Normal-1536.png,Normal,Normal/images/Normal-1536.png,Normal/masks/Normal-1536.png,True,G10204,train,fed9f1ab9895556f44b71446db2f4848,dd2c47034b4d4f5c
Normal-1537.png,Normal,Normal/images/Normal-1537.png,Normal/masks/Normal-1537.png,True,G10205,train,eeffb128605bda965883e34203b9f923,c23d2769e4c29b99
Normal-1538.png,Normal,Normal/images/Normal-1538.png,Normal/masks/Normal-1538.png,True,G10206,val,8814d82b18bcd4911035c08700f34186,d92946571f4e4a1d
Normal-1539.png,Normal,Normal/images/Normal-1539.png,Normal/masks/Normal-1539.png,True,G10207,train,a87c78b4d253779336bb4cc4dcd74617,817994161a5f59fa
Normal-154.png,Normal,Normal/images/Normal-154.png,Normal/masks/Normal-154.png,True,G10208,test,380ba59379664fc8a6fd8f0c6743ae53,d468e26be96868e5
Normal-1540.png,Normal,Normal/images/Normal-1540.png,Normal/masks/Normal-1540.png,True,G10209,val,b13442ca1b470aff356abd4afd29f688,d93dd21b5c0b3994
Normal-1541.png,Normal,Normal/images/Normal-1541.png,Normal/masks/Normal-1541.png,True,G10210,train,7aae24c932488827122697eac974fbcf,916897961a195f7a
Normal-1542.png,Normal,Normal/images/Normal-1542.png,Normal/masks/Normal-1542.png,True,G10211,train,b8031a2725c408087dd7152af73ee899,d97dc61a0d4c1b2d
Normal-1543.png,Normal,Normal/images/Normal-1543.png,Normal/masks/Normal-1543.png,True,G10212,val,1136fc95e4c97088367b41a87ea29053,cc7ea2531b41cb0f
Normal-1544.png,Normal,Normal/images/Normal-1544.png,Normal/masks/Normal-1544.png,True,G10213,train,14b99518e5c3b28a5fc1dde8fb9c8aff,9829121f597e3ae9
Normal-1545.png,Normal,Normal/images/Normal-1545.png,Normal/masks/Normal-1545.png,True,G10214,train,32f19a7bde95d1e3f7d657170212c76d,917a061e165e1f1f
Normal-1546.png,Normal,Normal/images/Normal-1546.png,Normal/masks/Normal-1546.png,True,G10215,train,384d2d881049f8ef64783641b79b56a0,992c061a5bce5af9
Normal-1547.png,Normal,Normal/images/Normal-1547.png,Normal/masks/Normal-1547.png,True,G10216,train,8ef318bea826c991ed2a52f53c8702c1,91694e46191ede79
Normal-1548.png,Normal,Normal/images/Normal-1548.png,Normal/masks/Normal-1548.png,True,G10217,test,335bab5d8c95a8731c621504e705b519,d93944c656134f4f
Normal-1549.png,Normal,Normal/images/Normal-1549.png,Normal/masks/Normal-1549.png,True,G10218,test,2cfb9b33c1a244b0606c40caf7580f0e,dd2a424f0a4f1c6f
Normal-155.png,Normal,Normal/images/Normal-155.png,Normal/masks/Normal-155.png,True,G10219,train,bec48e6bb2f4d4941dc0854bbbc11da1,91f0c6189dcd3ac7
Normal-1550.png,Normal,Normal/images/Normal-1550.png,Normal/masks/Normal-1550.png,True,G10220,train,23f8a6c0387a8fd2b434ead0854f9e2d,997b461e599d194a
Normal-1551.png,Normal,Normal/images/Normal-1551.png,Normal/masks/Normal-1551.png,True,G10221,train,5cd79f3dc86ce68901236081e380c262,d93c461a974b3da8
Normal-1552.png,Normal,Normal/images/Normal-1552.png,Normal/masks/Normal-1552.png,True,G10222,train,8c501ab67f9ef5cad6d87fd33301c880,d42e4251434f5f57
Normal-1553.png,Normal,Normal/images/Normal-1553.png,Normal/masks/Normal-1553.png,True,G10223,train,47db29ace278ab39ca7ae34b7f24d826,d93fc20b9dc08bb8
Normal-1554.png,Normal,Normal/images/Normal-1554.png,Normal/masks/Normal-1554.png,True,G10224,train,f7e866ea3ec25987854b0a830167f128,9979061e1a1efe38
Normal-1555.png,Normal,Normal/images/Normal-1555.png,Normal/masks/Normal-1555.png,True,G10225,train,20b6bc4fbb11a5eada1c37c16970fc06,8178d71a184f3b3b
Normal-1556.png,Normal,Normal/images/Normal-1556.png,Normal/masks/Normal-1556.png,True,G10226,train,1b66d4e04bcc772181955b2ab9d6b2d3,99f9c61a15c71e15
Normal-1557.png,Normal,Normal/images/Normal-1557.png,Normal/masks/Normal-1557.png,True,G10227,train,53f86d7300ba13cb0c78e5fd80c72de2,957d861a7969384d
Normal-1558.png,Normal,Normal/images/Normal-1558.png,Normal/masks/Normal-1558.png,True,G10228,train,00258c96ad32148e3312dd1088718418,c03dcb1e4b2d3c2d
Normal-1559.png,Normal,Normal/images/Normal-1559.png,Normal/masks/Normal-1559.png,True,G10229,train,345593a2d12f1750d8289a1fe380f9bf,d83dc21618c72fbc
Normal-156.png,Normal,Normal/images/Normal-156.png,Normal/masks/Normal-156.png,True,G10230,train,7d63c8711b394cb22f58fa8116648052,91e8c67ac969c9c6
Normal-1560.png,Normal,Normal/images/Normal-1560.png,Normal/masks/Normal-1560.png,True,G10231,train,68dd431ba10d4948f42a4ad516a9b6dd,913946065a5f6d7a
Normal-1561.png,Normal,Normal/images/Normal-1561.png,Normal/masks/Normal-1561.png,True,G10232,train,46b8d20f546d37adc2b2b0534d3063b0,8879d7125e157e19
Normal-1562.png,Normal,Normal/images/Normal-1562.png,Normal/masks/Normal-1562.png,True,G10233,val,c368f31430c14b0668db1c6aa7dcec68,9869071e5c5b7ae1
Normal-1563.png,Normal,Normal/images/Normal-1563.png,Normal/masks/Normal-1563.png,True,G10234,val,bc78ed32ecb9d68a3edd0b2afad09877,997cd61bc55a1945
Normal-1564.png,Normal,Normal/images/Normal-1564.png,Normal/masks/Normal-1564.png,True,G10235,val,d243b7e553742f6bca286de093d1c41d,dd2c46435b486f78
Normal-1565.png,Normal,Normal/images/Normal-1565.png,Normal/masks/Normal-1565.png,True,G10236,test,c168736e41feac8abf9b8e9631d4b166,d13d46461e1e4f5a
Normal-1566.png,Normal,Normal/images/Normal-1566.png,Normal/masks/Normal-1566.png,True,G10237,test,5e3aa1d0d7ae373388598c5786c1f337,d929459716471f1e
Normal-1567.png,Normal,Normal/images/Normal-1567.png,Normal/masks/Normal-1567.png,True,G10238,train,7144c64877003f1b5110670ef607714a,997bc51e19491a5e
Normal-1568.png,Normal,Normal/images/Normal-1568.png,Normal/masks/Normal-1568.png,True,G10239,test,9ad441c57e39a9ee0a5911f173d23638,dd2e02431fac6bb8
Normal-1569.png,Normal,Normal/images/Normal-1569.png,Normal/masks/Normal-1569.png,True,G10240,train,653bafc3075a2c635572fc0a26c1c124,d97b561a181b1b2b
Normal-157.png,Normal,Normal/images/Normal-157.png,Normal/masks/Normal-157.png,True,G10241,train,fa7211dc2039c96dfca443b02e05f3f5,9129e23af8693979
Normal-1570.png,Normal,Normal/images/Normal-1570.png,Normal/masks/Normal-1570.png,True,G10242,test,f939df6d5914d2dcba7841256455b370,887986c61e1fa65b
Normal-1571.png,Normal,Normal/images/Normal-1571.png,Normal/masks/Normal-1571.png,True,G10243,train,6681d086b65860d652213dd54071e70f,dd28d20b5d083ff4
Normal-1572.png,Normal,Normal/images/Normal-1572.png,Normal/masks/Normal-1572.png,True,G10244,train,5778aef33f699d365acddd04a79c8079,90676d1c78387b9c
Normal-1573.png,Normal,Normal/images/Normal-1573.png,Normal/masks/Normal-1573.png,True,G10245,train,f449a7f3d0f5bd8a2037d80ac0311b80,dd6fd61c5c0d180b
Normal-1574.png,Normal,Normal/images/Normal-1574.png,Normal/masks/Normal-1574.png,True,G10246,train,4671179b71de087d877ae7a710c1961b,897d871a19471f39
Normal-1575.png,Normal,Normal/images/Normal-1575.png,Normal/masks/Normal-1575.png,True,G10247,train,d7572362a588ec50bcac68efa985f0ac,817f161a1d3b39a9
Normal-1576.png,Normal,Normal/images/Normal-1576.png,Normal/masks/Normal-1576.png,True,G10248,train,0dfcffa4d14812e0b88833d4040b0590,d96f461b192f3832
Normal-1577.png,Normal,Normal/images/Normal-1577.png,Normal/masks/Normal-1577.png,True,G10249,train,5da1fdcbe55d4715e4d2c5b502a17c9b,817b56861b194f5b
Normal-1578.png,Normal,Normal/images/Normal-1578.png,Normal/masks/Normal-1578.png,True,G10250,train,46c31ecd7b9477fbb210ade821ddc845,8179061a1b5d5b7b
Normal-1579.png,Normal,Normal/images/Normal-1579.png,Normal/masks/Normal-1579.png,True,G10251,train,d43a0f9e2489fb3fc7a17be85b7b8842,d96c061a154f5b5d
Normal-158.png,Normal,Normal/images/Normal-158.png,Normal/masks/Normal-158.png,True,G10252,train,49d7cb9ee7c40c2d78c2925456295bc6,913be73a3a4e3838
Normal-1580.png,Normal,Normal/images/Normal-1580.png,Normal/masks/Normal-1580.png,True,G10253,test,d0c8925d017abc1ce1eb68fb337bbbb5,d93d86421f454f1d
Normal-1581.png,Normal,Normal/images/Normal-1581.png,Normal/masks/Normal-1581.png,True,G10254,train,d2f4956e50e1468613f1ac17a19321f5,897887165a4d5f1d
Normal-1582.png,Normal,Normal/images/Normal-1582.png,Normal/masks/Normal-1582.png,True,G10255,train,230a28dbf0e4fd7a48864bcb48c89e05,dd2e4257434b1b4b
Normal-1583.png,Normal,Normal/images/Normal-1583.png,Normal/masks/Normal-1583.png,True,G10256,val,7a30c33534c03ced06c810fc4bb9d26c,9879841e1949bff8
Normal-1584.png,Normal,Normal/images/Normal-1584.png,Normal/masks/Normal-1584.png,True,G10257,val,3dd82c5c9d06afed7909c360e03f22e5,d17bc51616c6561e
Normal-1585.png,Normal,Normal/images/Normal-1585.png,Normal/masks/Normal-1585.png,True,G10258,val,a1ec33d5318b3e14a9761ffbae7b2bb8,816886171adefeb8
Normal-1586.png,Normal,Normal/images/Normal-1586.png,Normal/masks/Normal-1586.png,True,G10259,test,d2336e1fcb53f39b090fb679a8434384,c13d465a1b4d3f38
Normal-1587.png,Normal,Normal/images/Normal-1587.png,Normal/masks/Normal-1587.png,True,G10260,train,0c0ed9a707027ea0b64920e4944f5e7c,997e16197b78a1a8
Normal-1588.png,Normal,Normal/images/Normal-1588.png,Normal/masks/Normal-1588.png,True,G10261,train,0f3a6c31888dd8251a1c73f57548511a,dd2c025b124f1efc
Normal-1589.png,Normal,Normal/images/Normal-1589.png,Normal/masks/Normal-1589.png,True,G10262,train,703883acbf232acc1d8b04633f74d4e4,b90d86121d4f1d5f
Normal-159.png,Normal,Normal/images/Normal-159.png,Normal/masks/Normal-159.png,True,G10263,train,58ae9b12f66f9775433865867475ab69,91a98e1ab9793959
Normal-1590.png,Normal,Normal/images/Normal-1590.png,Normal/masks/Normal-1590.png,True,G10264,train,97498d85a8dd95a1e001921aaa7d7018,912b021e5f1b1bde
Normal-1591.png,Normal,Normal/images/Normal-1591.png,Normal/masks/Normal-1591.png,True,G10265,train,d44e3b7db1fb17e2132c7dacc44c973e,997906161a2ded3b
Normal-1592.png,Normal,Normal/images/Normal-1592.png,Normal/masks/Normal-1592.png,True,G10266,train,0ff9d7ff24646d8654efb7bcedbbee02,997d871a154f1955
Normal-1593.png,Normal,Normal/images/Normal-1593.png,Normal/masks/Normal-1593.png,True,G10267,train,0b0ee3a5c1ea6b4a959429710b4238b8,f97946561a185f49
Normal-1594.png,Normal,Normal/images/Normal-1594.png,Normal/masks/Normal-1594.png,True,G10268,train,b2a1c363532a2df33249c950f9202667,9179ce1e594f2938
Normal-1595.png,Normal,Normal/images/Normal-1595.png,Normal/masks/Normal-1595.png,True,G10269,train,7a3ff6b3056327f796e7e288be938825,d9afc21fd04f3848
Normal-1596.png,Normal,Normal/images/Normal-1596.png,Normal/masks/Normal-1596.png,True,G10270,train,a5af14782c554962b4c41cee5f140a03,dd2cd5521fc4471a
Normal-1597.png,Normal,Normal/images/Normal-1597.png,Normal/masks/Normal-1597.png,True,G10271,train,6fa1a9c91edd799d1ca8cf7fcc7d4585,997d871a1b8d1b29
Normal-1598.png,Normal,Normal/images/Normal-1598.png,Normal/masks/Normal-1598.png,True,G10272,val,3f8384c633e0d0ac935fddeb5f36b6bd,894d061e1a5f5b5b
Normal-1599.png,Normal,Normal/images/Normal-1599.png,Normal/masks/Normal-1599.png,True,G10273,train,3fa3d3dce106acb4535987f68d3d2cf1,d92cc21b154f1d5d
Normal-16.png,Normal,Normal/images/Normal-16.png,Normal/masks/Normal-16.png,True,G10274,test,a4526acb3d93ca521eca1000a2031ff1,f979c6161a4b1e1a
Normal-160.png,Normal,Normal/images/Normal-160.png,Normal/masks/Normal-160.png,True,G10275,val,9c2747dc5037ef14f1445c9c00180c0a,d464c63ead2d3c4c
Normal-1600.png,Normal,Normal/images/Normal-1600.png,Normal/masks/Normal-1600.png,True,G10276,test,7c4f507e0bfa556d3ee6ae06218bb3d3,9968c61a19c67abb
Normal-1601.png,Normal,Normal/images/Normal-1601.png,Normal/masks/Normal-1601.png,True,G10277,val,053cab85cc058c1605b70097e60c738f,dd2a87461f064e1f
Normal-1602.png,Normal,Normal/images/Normal-1602.png,Normal/masks/Normal-1602.png,True,G10278,train,a6d56ecc4b6e5beb560b5e627224fd05,dd3ec20b1c4c3be8
Normal-1603.png,Normal,Normal/images/Normal-1603.png,Normal/masks/Normal-1603.png,True,G10279,test,97c317cf7718d70db23c96a4b2e98e49,897f841e4b555355
Normal-1604.png,Normal,Normal/images/Normal-1604.png,Normal/masks/Normal-1604.png,True,G10280,train,537e084873ed23846bef4a5b59e9ec3e,c93bc21f1c4e1b39
Normal-1605.png,Normal,Normal/images/Normal-1605.png,Normal/masks/Normal-1605.png,True,G10281,val,4afc2c2e857a3ed05e71313b9d06234b,c93dc20b944f1f93
Normal-1606.png,Normal,Normal/images/Normal-1606.png,Normal/masks/Normal-1606.png,True,G10282,train,7b03b3a2714a15828410fc7a52f1e89c,dc3d534f0e4a0d6c
Normal-1607.png,Normal,Normal/images/Normal-1607.png,Normal/masks/Normal-1607.png,True,G10283,train,46b2ed69d63bd1301ef06ff559bb50e4,c82dd70bd50b154f
Normal-1608.png,Normal,Normal/images/Normal-1608.png,Normal/masks/Normal-1608.png,True,G10284,train,36a7858c5e594fe66b04f938b91d190a,817a571a1f0f3b0b
Normal-1609.png,Normal,Normal/images/Normal-1609.png,Normal/masks/Normal-1609.png,True,G10285,train,be087ed8e5cb6859d1778c83f7a3dc0b,9168071e9a4b79fc
Normal-161.png,Normal,Normal/images/Normal-161.png,Normal/masks/Normal-161.png,True,G10286,train,b54c7e1a731841b2de0ecb4d3e21a515,8d0e0759cd61e3c7
Normal-1610.png,Normal,Normal/images/Normal-1610.png,Normal/masks/Normal-1610.png,True,G10287,train,08fcb8b0936e9eb480830dac6a6142c4,917946161a5f6979
Normal-1611.png,Normal,Normal/images/Normal-1611.png,Normal/masks/Normal-1611.png,True,G10288,train,d9123c07b4fa006577d564338bef9172,d42ec30f1d4d6d1a
Normal-1612.png,Normal,Normal/images/Normal-1612.png,Normal/masks/Normal-1612.png,True,G10289,train,3afc991029e1c8e73c043ab5e24cbd18,d92dc61b1c0639fc
Normal-1613.png,Normal,Normal/images/Normal-1613.png,Normal/masks/Normal-1613.png,True,G10290,train,091bb0a4e40b860f972611eeaa403585,f978c69b198c8e38
Normal-1614.png,Normal,Normal/images/Normal-1614.png,Normal/masks/Normal-1614.png,True,G10291,train,7ac317c5bfb10702c97e105757888d9c,897d061a1e4c7af9
Normal-1615.png,Normal,Normal/images/Normal-1615.png,Normal/masks/Normal-1615.png,True,G10292,train,4af70c5b9d9857d8173a935eff6277f9,dd28531f461d514f
Normal-1616.png,Normal,Normal/images/Normal-1616.png,Normal/masks/Normal-1616.png,True,G10293,train,3af51a7b6610785ec3c4f34d7846d848,d42b1f1d4d1c4b4b
Normal-1617.png,Normal,Normal/images/Normal-1617.png,Normal/masks/Normal-1617.png,True,G10294,train,8307daf5a0d0ca4c44d15cdf2528c7b7,d87b05c65a1d4d4d
Normal-1618.png,Normal,Normal/images/Normal-1618.png,Normal/masks/Normal-1618.png,True,G10295,val,f4d1c6a50b9f1e5950d7036926191c57,887f041e195d5b1f
Normal-1619.png,Normal,Normal/images/Normal-1619.png,Normal/masks/Normal-1619.png,True,G10296,val,132ecc5ceaad344b6bdc61995865964b,d568565e585a5879
Normal-162.png,Normal,Normal/images/Normal-162.png,Normal/masks/Normal-162.png,True,G10297,test,502971a68cd932dba986d7ef795471cd,916f861acd1d49c7
Normal-1620.png,Normal,Normal/images/Normal-1620.png,Normal/masks/Normal-1620.png,True,G10298,test,5dc333069a643419159ed4079df8ce71,d43d82431bd34b2f
Normal-1621.png,Normal,Normal/images/Normal-1621.png,Normal/masks/Normal-1621.png,True,G10299,test,b102f1fe7da4399ebc55b5b2a3848227,d92f461b1a0e39ba
Normal-1622.png,Normal,Normal/images/Normal-1622.png,Normal/masks/Normal-1622.png,True,G10300,train,e8ff1bec8939449924f094682cc7d75b,d97b061a1d5b194b
Normal-1623.png,Normal,Normal/images/Normal-1623.png,Normal/masks/Normal-1623.png,True,G10301,train,e02478668d058ea6c183ac9803e03379,d97bc61e1a4e1a19
Normal-1624.png,Normal,Normal/images/Normal-1624.png,Normal/masks/Normal-1624.png,True,G10302,train,7613f36e09d69b2664e596f6a3f1499f,dd3ec24b1f444a3c
Normal-1625.png,Normal,Normal/images/Normal-1625.png,Normal/masks/Normal-1625.png,True,G10303,train,5c354fa98a2dbb7cfbd52ecc47cb1bbf,c87d43430b0f4f2f
Normal-1626.png,Normal,Normal/images/Normal-1626.png,Normal/masks/Normal-1626.png,True,G10304,train,df72bde0147b6422dff641b090022747,817fc61a596d3887
Normal-1627.png,Normal,Normal/images/Normal-1627.png,Normal/masks/Normal-1627.png,True,G10305,train,fe09777730d7662bd2e3d31f5a23a2e6,987c0e0f1b6e6c39
Normal-1628.png,Normal,Normal/images/Normal-1628.png,Normal/masks/Normal-1628.png,True,G10306,train,b0b8c97c242b13c9f99a235d95ade68b,9d3d820b194d4e7d
Normal-1629.png,Normal,Normal/images/Normal-1629.png,Normal/masks/Normal-1629.png,True,G10307,train,6b57868f3d1b62424aa3e77b42810685,dd2a565b1e4d0b19
Normal-163.png,Normal,Normal/images/Normal-163.png,Normal/masks/Normal-163.png,True,G10308,train,5c7a46bbde112acce0e89d639a96bd11,917fcf1a19491cd8
Normal-1630.png,Normal,Normal/images/Normal-1630.png,Normal/masks/Normal-1630.png,True,G10309,train,30f1254d8e3d32c6013bc755241f046a,dd28d00b1f4e0f3d
Normal-1631.png,Normal,Normal/images/Normal-1631.png,Normal/masks/Normal-1631.png,True,G10310,train,f0f31d2ae0c83ecd1edca4bb13108ab7,8c474b0c6e3ce4bd
Normal-1632.png,Normal,Normal/images/Normal-1632.png,Normal/masks/Normal-1632.png,True,G10311,train,e3688dec530aaea7f99a8637e0ae8f76,dd2a530f460f1e4b
Normal-1633.png,Normal,Normal/images/Normal-1633.png,Normal/masks/Normal-1633.png,True,G10312,val,2a512a2e95b32e3313da69832401fdaf,91794e0e114e4f7b
Normal-1634.png,Normal,Normal/images/Normal-1634.png,Normal/masks/Normal-1634.png,True,G10313,val,e80325c2e42fb127f39aa29b9a644fb5,fd78d3431e404b1e
Normal-1635.png,Normal,Normal/images/Normal-1635.png,Normal/masks/Normal-1635.png,True,G10314,train,d357d4bd6596b2b7913bd29b9ccd381d,dd02530b4e1f4f2b
Normal-1636.png,Normal,Normal/images/Normal-1636.png,Normal/masks/Normal-1636.png,True,G10315,val,54802a1d7512f39573d23b636800a2c8,b96dc61a471a594d
Normal-1637.png,Normal,Normal/images/Normal-1637.png,Normal/masks/Normal-1637.png,True,G10316,test,cc567fd3d446b0345c7efee33cec0ae2,dd2a535b421f486e
Normal-1638.png,Normal,Normal/images/Normal-1638.png,Normal/masks/Normal-1638.png,True,G10317,train,5c6b7157051d9d6ed3c1fe95d4db71be,dd3dd257184e4d18
Normal-1639.png,Normal,Normal/images/Normal-1639.png,Normal/masks/Normal-1639.png,True,G10318,train,2a689aba9b334f3227861ee778a2ba9c,d978061a79e47939
Normal-164.png,Normal,Normal/images/Normal-164.png,Normal/masks/Normal-164.png,True,G10319,test,b3b7a3e6ec01b220a68224aabc6913ab,913ad238d07796d6
Normal-1640.png,Normal,Normal/images/Normal-1640.png,Normal/masks/Normal-1640.png,True,G10320,test,0db844bb498fccce00f4edc73e3bc66a,cd3e534f1a403f19
Normal-1641.png,Normal,Normal/images/Normal-1641.png,Normal/masks/Normal-1641.png,True,G10321,train,cb09bfa01a543b8de1504f1675fe9295,8376841d959f3a1d
Normal-1642.png,Normal,Normal/images/Normal-1642.png,Normal/masks/Normal-1642.png,True,G10322,test,8ee22f32c3a3795aeec4e61112c07981,8179c61e1d5ee83a
Normal-1643.png,Normal,Normal/images/Normal-1643.png,Normal/masks/Normal-1643.png,True,G10323,train,fd8035b44759571cd84522fe6514db3b,c93dc25b96c2955a
Normal-1644.png,Normal,Normal/images/Normal-1644.png,Normal/masks/Normal-1644.png,True,G10324,train,e5201bcae8423ac64d136af8d7507d6b,817905165a5d5d7b
Normal-1645.png,Normal,Normal/images/Normal-1645.png,Normal/masks/Normal-1645.png,True,G10325,train,729e16803c22a4c5897834fc71807080,912c5eda2ad929c7
Normal-1646.png,Normal,Normal/images/Normal-1646.png,Normal/masks/Normal-1646.png,True,G10326,train,cc31f36884ba658c706c2fc1e5234e21,f92dc61a154f3949
Normal-1647.png,Normal,Normal/images/Normal-1647.png,Normal/masks/Normal-1647.png,True,G10327,train,85f768f789b2b906074a199117e573a1,9969c61ec51b594d
Normal-1648.png,Normal,Normal/images/Normal-1648.png,Normal/masks/Normal-1648.png,True,G10328,train,c5c6ed57773fb0b2dc470b12387f31c4,996c871a4c7e79a4
Normal-1649.png,Normal,Normal/images/Normal-1649.png,Normal/masks/Normal-1649.png,True,G10329,train,2772e65799780d9273035effaf406871,952e07596e68c6f8
Normal-165.png,Normal,Normal/images/Normal-165.png,Normal/masks/Normal-165.png,True,G10330,train,7b40de8f018992bf8121818737fe4480,917f8f1a0d0d58c7
Normal-1650.png,Normal,Normal/images/Normal-1650.png,Normal/masks/Normal-1650.png,True,G10331,train,a5b29ccdf6b6ae2e02a24d5a2b20be95,9c2cd31b594d4c5d
Normal-1651.png,Normal,Normal/images/Normal-1651.png,Normal/masks/Normal-1651.png,True,G10332,train,4c1745f9dd3566a70af4482243c012b4,d43dc6431f1cc61b
Normal-1652.png,Normal,Normal/images/Normal-1652.png,Normal/masks/Normal-1652.png,True,G10333,train,3f0288b4a1c9c57914ef689b3df5d54d,d42a431b4b6d3d6c
Normal-1653.png,Normal,Normal/images/Normal-1653.png,Normal/masks/Normal-1653.png,True,G10334,val,c246d6ade666e88b0abdcee2a4dcb4c9,dd38d21fd20f180f
Normal-1654.png,Normal,Normal/images/Normal-1654.png,Normal/masks/Normal-1654.png,True,G10335,val,d5444ae4fec2c72f32a3c3c2eb5dbd91,813c43c31f5bec1b
Normal-1655.png,Normal,Normal/images/Normal-1655.png,Normal/masks/Normal-1655.png,True,G10336,val,9478d78ec3cba0c87c3bb8d9e4920e8c,c03d43434d1acf6f
Normal-1656.png,Normal,Normal/images/Normal-1656.png,Normal/masks/Normal-1656.png,True,G10337,test,03982e08900e7215f73f2223b8e3c003,c9389e1b7d6839e0
Normal-1657.png,Normal,Normal/images/Normal-1657.png,Normal/masks/Normal-1657.png,True,G10338,test,b56dd5b88e571af8660a05bcfad121b9,995dc65a1b2d2c39
Normal-1658.png,Normal,Normal/images/Normal-1658.png,Normal/masks/Normal-1658.png,True,G10339,train,35320271957fa3000e7ac4797acf7239,8c6e0312594d7f7c
Normal-1659.png,Normal,Normal/images/Normal-1659.png,Normal/masks/Normal-1659.png,True,G10340,train,0b40d87b75d8aacbf1541abdbc432488,c13f7a8467c93cc2
Normal-166.png,Normal,Normal/images/Normal-166.png,Normal/masks/Normal-166.png,True,G10341,test,450376ceb8bc1098f188ff85159dc544,c06ce66b3ded2d20
Normal-1660.png,Normal,Normal/images/Normal-1660.png,Normal/masks/Normal-1660.png,True,G10342,train,85c7bd8d4a448a82907dec746857e889,f961961e501bc51f
Normal-1661.png,Normal,Normal/images/Normal-1661.png,Normal/masks/Normal-1661.png,True,G10343,train,25ea9f177dffbd0bd475a940d1a475ad,d13dc7425d1d4a59
Normal-1662.png,Normal,Normal/images/Normal-1662.png,Normal/masks/Normal-1662.png,True,G10344,train,9abc630e2046357c7cd581ff2b06ed78,dd2c52831fc68e0f
Normal-1663.png,Normal,Normal/images/Normal-1663.png,Normal/masks/Normal-1663.png,True,G10345,train,4366a6a6750805657761ef9ef0dfd664,987f465a5d4a5949
Normal-1664.png,Normal,Normal/images/Normal-1664.png,Normal/masks/Normal-1664.png,True,G10346,train,1a966fb25fbf0b720fdbc9600ed17e3f,956886861a5f9e5e
Normal-1665.png,Normal,Normal/images/Normal-1665.png,Normal/masks/Normal-1665.png,True,G10347,train,bb19c03defd62585788452900336d3c5,9169871e1c5d7ab8
Normal-1666.png,Normal,Normal/images/Normal-1666.png,Normal/masks/Normal-1666.png,True,G10348,train,567a1ddec4896daef06ac7ea590a75a4,817dc35b5a4a5b0b
Normal-1667.png,Normal,Normal/images/Normal-1667.png,Normal/masks/Normal-1667.png,True,G10349,train,704c8bbe4a4e47be52ce67df031ace8e,907e462365e7e398
Normal-1668.png,Normal,Normal/images/Normal-1668.png,Normal/masks/Normal-1668.png,True,G10350,train,b4e0f299d2923bdb0dcc9acc0ae4a55f,d939465f1a4d4e1a
Normal-1669.png,Normal,Normal/images/Normal-1669.png,Normal/masks/Normal-1669.png,True,G10351,train,1c58ec56e3f1ed0b724f1ee9b1fe47cc,a97d061a194d7bf0
Normal-167.png,Normal,Normal/images/Normal-167.png,Normal/masks/Normal-167.png,True,G10352,train,6b216309b68a3e4b6d225c2c6a6d6f13,916b4e1a5d4d1cc7
Normal-1670.png,Normal,Normal/images/Normal-1670.png,Normal/masks/Normal-1670.png,True,G10353,val,2d627f1f1e6eac49a6e1be8efb70524a,d82dc2d71fc24b1c
Normal-1671.png,Normal,Normal/images/Normal-1671.png,Normal/masks/Normal-1671.png,True,G10354,train,a9da640fc085455f99995bfa25c487f2,8178071a1f5f7a39
Normal-1672.png,Normal,Normal/images/Normal-1672.png,Normal/masks/Normal-1672.png,True,G10355,train,dcf9fa05ec3c91e131afaaaf68f7e314,893cc60f19475d79
Normal-1673.png,Normal,Normal/images/Normal-1673.png,Normal/masks/Normal-1673.png,True,G10356,test,7e513b6854befad1e04b1d49a0927b87,fd3b47561949491c
Normal-1674.png,Normal,Normal/images/Normal-1674.png,Normal/masks/Normal-1674.png,True,G10357,val,3ba1f37003d84e0911b640d24c46bca1,8907065b4d7acd7a
Normal-1675.png,Normal,Normal/images/Normal-1675.png,Normal/masks/Normal-1675.png,True,G10358,test,4df38b245dbe06e9a634aa319ff639d3,897d061e1ce979b8
Normal-1676.png,Normal,Normal/images/Normal-1676.png,Normal/masks/Normal-1676.png,True,G10359,val,338e39af9de0e42d541cdb323aa961bd,dc2e42cb0aed0bbc
Normal-1677.png,Normal,Normal/images/Normal-1677.png,Normal/masks/Normal-1677.png,True,G10360,train,cf48d59b8ace182ba4920dde8fa865c5,dd2942570e554b1f
Normal-1678.png,Normal,Normal/images/Normal-1678.png,Normal/masks/Normal-1678.png,True,G10361,train,1812350b48f4bcc845c4bba73e2a2d6e,9978865a095e7bc5
Normal-1679.png,Normal,Normal/images/Normal-1679.png,Normal/masks/Normal-1679.png,True,G10362,train,9cc9724c14595bb8334eda18527e1ee8,992d47061a0dff5a
Normal-168.png,Normal,Normal/images/Normal-168.png,Normal/masks/Normal-168.png,True,G10363,train,2e9e1b270bc6173c04363d67ff939778,907ac67ab8653db0
Normal-1680.png,Normal,Normal/images/Normal-1680.png,Normal/masks/Normal-1680.png,True,G10364,train,63805127b4b3fa43bcef6bfaac87ee33,c039cb1f586f3d90
Normal-1681.png,Normal,Normal/images/Normal-1681.png,Normal/masks/Normal-1681.png,True,G10365,train,0d023156937c7f6840e3c217a32b82af,817886195f7f3952
Normal-1682.png,Normal,Normal/images/Normal-1682.png,Normal/masks/Normal-1682.png,True,G10366,train,8b65bc1c98d175ef1c01544185283577,d938d6431f0e4b1d
Normal-1683.png,Normal,Normal/images/Normal-1683.png,Normal/masks/Normal-1683.png,True,G10367,train,3954f6bcdeef4710cf57e9d6e5866b71,9979851a5f4d5878
Normal-1684.png,Normal,Normal/images/Normal-1684.png,Normal/masks/Normal-1684.png,True,G10368,val,2297083c5a79252687006d1bb0691b53,dc2ec34f4c4a6ce2
Normal-1685.png,Normal,Normal/images/Normal-1685.png,Normal/masks/Normal-1685.png,True,G10369,train,2d3fee9d92e57c5066921d9a10ea6fca,91299e197b5ee906
Normal-1686.png,Normal,Normal/images/Normal-1686.png,Normal/masks/Normal-1686.png,True,G10370,train,14fac9b3d76498c3f36b1fd4ba69081a,c0adc20f1f80afbe
Normal-1687.png,Normal,Normal/images/Normal-1687.png,Normal/masks/Normal-1687.png,True,G10371,train,7eaf214315abbbda6a057c0c2b43b5b0,893dc41e1f451a5f
Normal-1688.png,Normal,Normal/images/Normal-1688.png,Normal/masks/Normal-1688.png,True,G10372,train,df8b36fa39703b40d4a1863af0d3a416,dd2cc31b1d402f3d
Normal-1689.png,Normal,Normal/images/Normal-1689.png,Normal/masks/Normal-1689.png,True,G10373,train,4402736cc4e35706e1a4fa11def75bfa,dc2cc31b450f1d4f
Normal-169.png,Normal,Normal/images/Normal-169.png,Normal/masks/Normal-169.png,True,G10374,train,86dd22e1256e209d68e18a09644ee492,c02fc70db8c33bb2
Normal-1690.png,Normal,Normal/images/Normal-1690.png,Normal/masks/Normal-1690.png,True,G10375,train,5eb287733f6d00a5e866c0e890291136,d02f820f1d554f1f
Normal-1691.png,Normal,Normal/images/Normal-1691.png,Normal/masks/Normal-1691.png,True,G10376,val,00ddab9d23a09d3f3978785034138c83,815f86187f4e39e2
Normal-1692.png,Normal,Normal/images/Normal-1692.png,Normal/masks/Normal-1692.png,True,G10377,val,f24c5b916264e472ff0270ccb59a83b7,997987579817861e
Normal-1693.png,Normal,Normal/images/Normal-1693.png,Normal/masks/Normal-1693.png,True,G10378,test,02c390de0a1878748eb90decd2f34af0,c969c61bd44f5945
Normal-1694.png,Normal,Normal/images/Normal-1694.png,Normal/masks/Normal-1694.png,True,G10379,test,ce02c42c1acff9959330b348daf7dea6,d92847171e4d2e3d
Normal-1695.png,Normal,Normal/images/Normal-1695.png,Normal/masks/Normal-1695.png,True,G10380,train,c16e6e60107d7b39424242dde7f34d86,9169861a1b5e79f1
Normal-1696.png,Normal,Normal/images/Normal-1696.png,Normal/masks/Normal-1696.png,True,G10381,test,fd86e02b9518e6ef4f668dc3f696934c,882f061f145f1f5b
Normal-1697.png,Normal,Normal/images/Normal-1697.png,Normal/masks/Normal-1697.png,True,G10382,train,4c309d0a90e95aba0afa1c884d2a96f2,91285e5b5b5b5a1a
Normal-1698.png,Normal,Normal/images/Normal-1698.png,Normal/masks/Normal-1698.png,True,G10383,train,34b510e0d5706e73379b0a437c742313,d97dc61e114b19a9
Normal-1699.png,Normal,Normal/images/Normal-1699.png,Normal/masks/Normal-1699.png,True,G10384,train,164fb91f695840915fc503c861d75087,d979061e186e7b61
Normal-17.png,Normal,Normal/images/Normal-17.png,Normal/masks/Normal-17.png,True,G10385,train,89264e6c59134ccf009fe354f388f1ae,9d6c874719c74d38
Normal-170.png,Normal,Normal/images/Normal-170.png,Normal/masks/Normal-170.png,True,G10386,train,e2f9fe6492d1e36b0898c3bac1d794db,d03eab3f8c8a1a96
Normal-1700.png,Normal,Normal/images/Normal-1700.png,Normal/masks/Normal-1700.png,True,G10387,val,009702ce5ed03e5d35f4ea4b28a45943,919178de5a399565
Normal-1701.png,Normal,Normal/images/Normal-1701.png,Normal/masks/Normal-1701.png,True,G10388,train,ac033628bc4b048b42aecfcb1a8d918d,91780716185f5f7a
Normal-1702.png,Normal,Normal/images/Normal-1702.png,Normal/masks/Normal-1702.png,True,G10389,train,a2927ff0212fe99fc115f5671f51a743,dd2e470b1d465b1c
Normal-1703.png,Normal,Normal/images/Normal-1703.png,Normal/masks/Normal-1703.png,True,G10390,train,0cb90224ae0f358992e018f9f079face,997c42191e6d39e5
Normal-1704.png,Normal,Normal/images/Normal-1704.png,Normal/masks/Normal-1704.png,True,G10391,train,56679313bb88a81ea13862aab50d83b9,81695615585f6d79
Normal-1705.png,Normal,Normal/images/Normal-1705.png,Normal/masks/Normal-1705.png,True,G10392,train,de9925ce17c6b5a3d7ac70b72fb86a22,d674274b4f082f0f
Normal-1706.png,Normal,Normal/images/Normal-1706.png,Normal/masks/Normal-1706.png,True,G10393,train,367e219346ba5201f0672de647e203e1,c93d421f8a0f3da9
Normal-1707.png,Normal,Normal/images/Normal-1707.png,Normal/masks/Normal-1707.png,True,G10394,val,d92b0a5dd73c67ad42a8c05529c4f334,9ca9c38fe59194e1
Normal-1708.png,Normal,Normal/images/Normal-1708.png,Normal/masks/Normal-1708.png,True,G10395,val,2093eea18df33375d56461d9c6ad7d53,dd2cc25b154b1f4c
Normal-1709.png,Normal,Normal/images/Normal-1709.png,Normal/masks/Normal-1709.png,True,G10396,val,feabc88f633b33e3bcd7441f346c3cda,d17952425e185d6f
Normal-171.png,Normal,Normal/images/Normal-171.png,Normal/masks/Normal-171.png,True,G10397,test,52509a12ef0fbd07f69fef27d7a6a38e,91fcc71ac92d7984
Normal-1710.png,Normal,Normal/images/Normal-1710.png,Normal/masks/Normal-1710.png,True,G10398,test,cf428852b03b6e878ef0c48c0cee62a6,887f06185f5a7c4e
Normal-1711.png,Normal,Normal/images/Normal-1711.png,Normal/masks/Normal-1711.png,True,G10399,train,97b9b2fc5742e0733168e4ae828357fb,c034bf43b10bcbae
Normal-1712.png,Normal,Normal/images/Normal-1712.png,Normal/masks/Normal-1712.png,True,G10400,test,00ef6baa1b730a8c58dabd26d867f8fa,997985171a4c7b3a
Normal-1713.png,Normal,Normal/images/Normal-1713.png,Normal/masks/Normal-1713.png,True,G10401,train,9dc3a5a7c0d2db106031ed34c11c2a6a,817bc7161a4e5e1b
Normal-1714.png,Normal,Normal/images/Normal-1714.png,Normal/masks/Normal-1714.png,True,G10402,train,f974de3eafc29ad1918e2f1ba5a606c6,8979065e194d7fd0
Normal-1715.png,Normal,Normal/images/Normal-1715.png,Normal/masks/Normal-1715.png,True,G10403,train,ca801d7ce6dc828d10746034b0249f19,d9ab4615471f105f
Normal-1716.png,Normal,Normal/images/Normal-1716.png,Normal/masks/Normal-1716.png,True,G10404,train,a3c0d282b47de5a72f65784faa51410e,b90d461b594c78f9
Normal-1717.png,Normal,Normal/images/Normal-1717.png,Normal/masks/Normal-1717.png,True,G10405,train,72420fdaaadb6cd2fa27096a99aaed7d,997d861a0e461afd
Normal-1718.png,Normal,Normal/images/Normal-1718.png,Normal/masks/Normal-1718.png,True,G10406,train,4f42bc39437daed021443d678870a96f,dd2b00871a1f4f1f
Normal-1719.png,Normal,Normal/images/Normal-1719.png,Normal/masks/Normal-1719.png,True,G10407,train,86620ffcdfe4c7910a7b6dff07854ccc,997b06161a4d3f39
Normal-172.png,Normal,Normal/images/Normal-172.png,Normal/masks/Normal-172.png,True,G10408,train,66f276ac32282cab46fe7b7fa451e5d4,906de73a894f38cc
Normal-1720.png,Normal,Normal/images/Normal-1720.png,Normal/masks/Normal-1720.png,True,G10409,train,28015ccf8b30c6d719b1be183d6d9dfd,dd2fd20b1ec30b0d
Normal-1721.png,Normal,Normal/images/Normal-1721.png,Normal/masks/Normal-1721.png,True,G10410,train,f00f12cf29dba53a2a3bac25de69a5de,993887165a4d5ef8
Normal-1722.png,Normal,Normal/images/Normal-1722.png,Normal/masks/Normal-1722.png,True,G10411,train,955b252953bcc6453104936984320d97,942ec61b5dece438
Normal-1723.png,Normal,Normal/images/Normal-1723.png,Normal/masks/Normal-1723.png,True,G10412,train,391c6e4cb942eb0a420951a4c9cb4cbe,81794e1b5a2d393d
Normal-1724.png,Normal,Normal/images/Normal-1724.png,Normal/masks/Normal-1724.png,True,G10413,train,b5552cce98ab5bee6118eb37391277c5,cc2fc24b5b430f1e
Normal-1725.png,Normal,Normal/images/Normal-1725.png,Normal/masks/Normal-1725.png,True,G10414,train,d3b9dfe1121db15390be1dfa98eba02a,d87b45c65e124a5b
Normal-1726.png,Normal,Normal/images/Normal-1726.png,Normal/masks/Normal-1726.png,True,G10415,val,ef89076dedd707bd5eab7605445f88e0,8c6c85525b456f5e
Normal-1727.png,Normal,Normal/images/Normal-1727.png,Normal/masks/Normal-1727.png,True,G10416,train,6295e0a2b9a50df91630547ef7c02841,8179c61e055b5bc7
Normal-1728.png,Normal,Normal/images/Normal-1728.png,Normal/masks/Normal-1728.png,True,G10417,val,965c6e4c4281baca0bfb903e2529978e,917bd7025f085f19
Normal-1729.png,Normal,Normal/images/Normal-1729.png,Normal/masks/Normal-1729.png,True,G10418,test,5304308f32a5d101b5a10ce7ee4c9872,913947161e5b7d1a
Normal-173.png,Normal,Normal/images/Normal-173.png,Normal/masks/Normal-173.png,True,G10419,val,1c56acca5deb70fab932e80529150da6,d47ec70f8d2c2cc4
Normal-1730.png,Normal,Normal/images/Normal-1730.png,Normal/masks/Normal-1730.png,True,G10420,test,255a6e2950e79a6cd20e48006957f340,9d2e871b05173f07
Normal-1731.png,Normal,Normal/images/Normal-1731.png,Normal/masks/Normal-1731.png,True,G10421,test,39350ceca318169feda87e8774d517dc,d13dce081d4e6c3b
Normal-1732.png,Normal,Normal/images/Normal-1732.png,Normal/masks/Normal-1732.png,True,G10422,train,4e4ea2809a54f3868c12013fe37209ec,dd2ec259544b9c4b
Normal-1733.png,Normal,Normal/images/Normal-1733.png,Normal/masks/Normal-1733.png,True,G10423,train,b29f7e427d0b87a74dd87f6b54534596,d43cc3cb1f1c4c1d
Normal-1734.png,Normal,Normal/images/Normal-1734.png,Normal/masks/Normal-1734.png,True,G10424,train,70efa4f5f008b017b58a0ea67b00519d,c83d870713574f1b
Normal-1735.png,Normal,Normal/images/Normal-1735.png,Normal/masks/Normal-1735.png,True,G10425,train,78906da7bc4fc0563a7aaddd430d7aa2,d52c87831d0f4f1d
Normal-1736.png,Normal,Normal/images/Normal-1736.png,Normal/masks/Normal-1736.png,True,G10426,train,a0ef2fa3880567b7336c8dc856ca0de6,dc2e434b4a0f4a6f
Normal-1737.png,Normal,Normal/images/Normal-1737.png,Normal/masks/Normal-1737.png,True,G10427,train,87fe66d1aa57b5192e593e7b89c4c587,c83f074a1d4d6eb8
Normal-1738.png,Normal,Normal/images/Normal-1738.png,Normal/masks/Normal-1738.png,True,G10428,train,dada6b537036cca75fd788d7ff44c60b,8179851c16cf5f33
Normal-1739.png,Normal,Normal/images/Normal-1739.png,Normal/masks/Normal-1739.png,True,G10429,train,59390901f24cd6ada07168a6c7b22833,dd3d420b1b4f0e59
Normal-174.png,Normal,Normal/images/Normal-174.png,Normal/masks/Normal-174.png,True,G10430,train,6796f190db44c419682c9256f9ef5c64,806ac75edc4b39c3
Normal-1740.png,Normal,Normal/images/Normal-1740.png,Normal/masks/Normal-1740.png,True,G10431,train,51ac82ca2edf301d017711f4273df41f,993d861a194f7a39
Normal-1741.png,Normal,Normal/images/Normal-1741.png,Normal/masks/Normal-1741.png,True,G10432,train,36efb69646cc124640159ba26238afc8,dd2cc20b5a6c2ebc
Normal-1742.png,Normal,Normal/images/Normal-1742.png,Normal/masks/Normal-1742.png,True,G10433,train,efe58ddc5a38d533fb2157d03fd8ed0e,996e96590f1a0f1e
Normal-1743.png,Normal,Normal/images/Normal-1743.png,Normal/masks/Normal-1743.png,True,G10434,val,8c5c649782185b325a5c445c0fec0870,9969871e195e5a59
Normal-1744.png,Normal,Normal/images/Normal-1744.png,Normal/masks/Normal-1744.png,True,G10435,val,52352e684b3fa1be30a81dfbd66ae057,9939460638f9ed78
Normal-1745.png,Normal,Normal/images/Normal-1745.png,Normal/masks/Normal-1745.png,True,G10436,val,230deac37e509c90d0696ee6da31278f,8139ca19194f3dbe
Normal-1746.png,Normal,Normal/images/Normal-1746.png,Normal/masks/Normal-1746.png,True,G10437,test,27b16320413d30216dab71c36321416d,dd3a530b5e0a1b0f
Normal-1747.png,Normal,Normal/images/Normal-1747.png,Normal/masks/Normal-1747.png,True,G10438,test,920f0887ed96078fa5120d1c311e62b7,d16bc6179e025717
Normal-1748.png,Normal,Normal/images/Normal-1748.png,Normal/masks/Normal-1748.png,True,G10439,test,5a370aecc763a631b7e0651f464cc086,cd07470f4d4a4f6a
Normal-1749.png,Normal,Normal/images/Normal-1749.png,Normal/masks/Normal-1749.png,True,G10440,train,cae1af8879435b23cc0cf15331b00523,917d071a175f5a45
Normal-175.png,Normal,Normal/images/Normal-175.png,Normal/masks/Normal-175.png,True,G10441,train,e4b5f878b321f7af5fbc9345b030003e,916dc61a9e4b3d68
Normal-1750.png,Normal,Normal/images/Normal-1750.png,Normal/masks/Normal-1750.png,True,G10442,train,fdd0f4f90b71b0e87c04c9ad5f046731,9d24431f471f451f
Normal-1751.png,Normal,Normal/images/Normal-1751.png,Normal/masks/Normal-1751.png,True,G10443,train,2fa000b7695137a7904e2a5fad5490b8,817c9e5e59595a58
Normal-1752.png,Normal,Normal/images/Normal-1752.png,Normal/masks/Normal-1752.png,True,G10444,train,0e79036e2c8717d0fc675c63da58c54c,d129d6061f9e4a9b
Normal-1753.png,Normal,Normal/images/Normal-1753.png,Normal/masks/Normal-1753.png,True,G10445,train,8f1cd444644490707088393c313afb85,8979c75e1a4d5919
Normal-1754.png,Normal,Normal/images/Normal-1754.png,Normal/masks/Normal-1754.png,True,G10446,train,68000c55029172fa617902c34ef1bab2,897f861e195d194b
Normal-1755.png,Normal,Normal/images/Normal-1755.png,Normal/masks/Normal-1755.png,True,G10447,train,bfc59127ed9a9b218d1e0fc0e276e007,cc3cc3194a2e3de5
Normal-1756.png,Normal,Normal/images/Normal-1756.png,Normal/masks/Normal-1756.png,True,G10448,train,0af075ba58cf569aa035b2c9ca329c08,9b395b1a385e9692
Normal-1757.png,Normal,Normal/images/Normal-1757.png,Normal/masks/Normal-1757.png,True,G10449,test,5930e87af1c519111e6a95f6463fccd5,987a07161d4f3e6a
Normal-1758.png,Normal,Normal/images/Normal-1758.png,Normal/masks/Normal-1758.png,True,G10450,train,c6c1cdefaf9702765eb50d6b0c75493f,893d06565c69e979
Normal-1759.png,Normal,Normal/images/Normal-1759.png,Normal/masks/Normal-1759.png,True,G10451,train,bf46b1e3390f3e35b2c8a096c0e09084,c43c43492dadfc3c
Normal-176.png,Normal,Normal/images/Normal-176.png,Normal/masks/Normal-176.png,True,G10452,train,0938f34cc0ec72971ad33ae84f0ada60,9129c638d97939d3
Normal-1760.png,Normal,Normal/images/Normal-1760.png,Normal/masks/Normal-1760.png,True,G10453,train,4dbeec2e14643231924275632f4b73b6,d528071f0e9e2b8f
Normal-1761.png,Normal,Normal/images/Normal-1761.png,Normal/masks/Normal-1761.png,True,G10454,val,74d6f8845845203a210ff535407b8d59,9928165a5b5d78e9
Normal-1762.png,Normal,Normal/images/Normal-1762.png,Normal/masks/Normal-1762.png,True,G10455,val,bcc99dcd602aff8c35e0647800363da6,987dc65e145b1b51
Normal-1763.png,Normal,Normal/images/Normal-1763.png,Normal/masks/Normal-1763.png,True,G10456,train,89b10ed534abe79f4b4703ed8cfa5463,d42f434f1e4a4f4a
Normal-1764.png,Normal,Normal/images/Normal-1764.png,Normal/masks/Normal-1764.png,True,G10457,train,f6148eb0324961abd0d38a7aa4e0aa0e,8978065a4f3c39dd
Normal-1765.png,Normal,Normal/images/Normal-1765.png,Normal/masks/Normal-1765.png,True,G10458,test,da1670d97d61fc3897c0320bda0fedc5,d939c6461a1f4e1b
Normal-1766.png,Normal,Normal/images/Normal-1766.png,Normal/masks/Normal-1766.png,True,G10459,train,e024703cb245f2280fd4ce5cc8964c83,987b47161a455e7a
Normal-1767.png,Normal,Normal/images/Normal-1767.png,Normal/masks/Normal-1767.png,True,G10460,test,aca85edd7b1d93f20404e4ed79e6758a,817846175b6d653b
Normal-1768.png,Normal,Normal/images/Normal-1768.png,Normal/masks/Normal-1768.png,True,G10461,val,4da395a7786192bd3ba2c8255bbe497c,9d2c465f196c78b8
Normal-1769.png,Normal,Normal/images/Normal-1769.png,Normal/masks/Normal-1769.png,True,G10462,test,3f85569376ffc3b932b1dbc98de6ce27,897987061a57de5a
Normal-177.png,Normal,Normal/images/Normal-177.png,Normal/masks/Normal-177.png,True,G10463,train,1ce0b54d8bc8028c83387f1a33b0e1f6,913dc71b9c6939d0
Normal-1770.png,Normal,Normal/images/Normal-1770.png,Normal/masks/Normal-1770.png,True,G10464,train,caabfe9f7db5186e5c3cef01509d046d,8168165b5e7d7871
Normal-1771.png,Normal,Normal/images/Normal-1771.png,Normal/masks/Normal-1771.png,True,G10465,train,930677879648c8bf1cf43f1ff42162f6,c9fdc214427f524d
Normal-1772.png,Normal,Normal/images/Normal-1772.png,Normal/masks/Normal-1772.png,True,G10466,train,8cbd99aa52586b3f7616a4bde7f0ba16,dd6dc65b406e484d
Normal-1773.png,Normal,Normal/images/Normal-1773.png,Normal/masks/Normal-1773.png,True,G10467,train,85a95f71d1f307230b59cae4ffffaaa0,993b989a9586ce6c
Normal-1774.png,Normal,Normal/images/Normal-1774.png,Normal/masks/Normal-1774.png,True,G10468,train,9b9beb862c359edc77708bb3d17d14cc,d13e064919fcedb0
Normal-1775.png,Normal,Normal/images/Normal-1775.png,Normal/masks/Normal-1775.png,True,G10469,train,7bbbddc8b9071f7128dbbf414fe20eef,d4adc25f14441f5b
Normal-1776.png,Normal,Normal/images/Normal-1776.png,Normal/masks/Normal-1776.png,True,G10470,train,545a180caf97f1c15e17a229b89046a9,dd2dc21b0b8e0bbc
Normal-1777.png,Normal,Normal/images/Normal-1777.png,Normal/masks/Normal-1777.png,True,G10471,train,9e7ada4d2dc83a754aa876a7a1eef68b,997d521b810e3cee
Normal-1778.png,Normal,Normal/images/Normal-1778.png,Normal/masks/Normal-1778.png,True,G10472,train,f92e1560a84ae53938b62b685d4a9cfc,887885171f855f1f
Normal-1779.png,Normal,Normal/images/Normal-1779.png,Normal/masks/Normal-1779.png,True,G10473,train,c67a2c854a8ddb3a1e7eac7ad756982c,dd28d64b1f0c4f29
Normal-178.png,Normal,Normal/images/Normal-178.png,Normal/masks/Normal-178.png,True,G10474,train,f8586c65a9e6c484d8862839d77c4f5e,916cce58cd491dce
Normal-1780.png,Normal,Normal/images/Normal-1780.png,Normal/masks/Normal-1780.png,True,G10475,train,078326cdde6becfe30240dec504e18d4,dd7c46431b8c2c3d
Normal-1781.png,Normal,Normal/images/Normal-1781.png,Normal/masks/Normal-1781.png,True,G10476,test,e8202024d20a64ea9b3b709299572b7a,c829571e1f463a79
Normal-1782.png,Normal,Normal/images/Normal-1782.png,Normal/masks/Normal-1782.png,True,G10477,val,d3fedfcc08338a1611047ce71213f37e,9979065a594d7ad8
Normal-1783.png,Normal,Normal/images/Normal-1783.png,Normal/masks/Normal-1783.png,True,G10478,val,81234ef75ac754c432a365d21b29acf1,d939461b1da8b9e4
Normal-1784.png,Normal,Normal/images/Normal-1784.png,Normal/masks/Normal-1784.png,True,G10479,val,6bf28d680bda53c2b6bba95874441aa9,dd2cd04b95421f5e
Normal-1785.png,Normal,Normal/images/Normal-1785.png,Normal/masks/Normal-1785.png,True,G10480,test,8384531ef422d6401993b6e99cdce402,883f571ac52ab86b
Normal-1786.png,Normal,Normal/images/Normal-1786.png,Normal/masks/Normal-1786.png,True,G10481,train,2c592eca1903ac0874f04f942f047651,ac2c834b0b4f1f1f
Normal-1787.png,Normal,Normal/images/Normal-1787.png,Normal/masks/Normal-1787.png,True,G10482,train,968bf29f218f15b881ddf381211683cf,dd2ec20b49445efc
Normal-1788.png,Normal,Normal/images/Normal-1788.png,Normal/masks/Normal-1788.png,True,G10483,train,827a8c6da0404c5e6620fe342e84cfee,817b961a185f7b13
Normal-1789.png,Normal,Normal/images/Normal-1789.png,Normal/masks/Normal-1789.png,True,G10484,test,b57bc8f32e90d7a1c0e86575d0c00116,d92cc41f544f1bc9
Normal-179.png,Normal,Normal/images/Normal-179.png,Normal/masks/Normal-179.png,True,G10485,train,d68c6412cf25d9ae91db27e505cb146f,9129c71abcc91d9d
Normal-1790.png,Normal,Normal/images/Normal-1790.png,Normal/masks/Normal-1790.png,True,G10486,train,f7516a69e63bd873e60103cd43610139,d82a57171f492abc
Normal-1791.png,Normal,Normal/images/Normal-1791.png,Normal/masks/Normal-1791.png,True,G10487,train,73f63ade52b1b2bcdeab68b1ccbb4758,9968061a5f5f394d
Normal-1792.png,Normal,Normal/images/Normal-1792.png,Normal/masks/Normal-1792.png,True,G10488,train,06edf4e1de1e88b7c60d96b5c797f13a,917846d71bc67a94
Normal-1793.png,Normal,Normal/images/Normal-1793.png,Normal/masks/Normal-1793.png,True,G10489,train,26571685bf44f47e46177e16c768f076,953ec65b1c606979
Normal-1794.png,Normal,Normal/images/Normal-1794.png,Normal/masks/Normal-1794.png,True,G10490,train,5845308808ffe78400e5e65daa5294d6,987d861a195f385b
Normal-1795.png,Normal,Normal/images/Normal-1795.png,Normal/masks/Normal-1795.png,True,G10491,train,7e8680050f950576e004ddcbdebecd09,907bc7865a594d59
Normal-1796.png,Normal,Normal/images/Normal-1796.png,Normal/masks/Normal-1796.png,True,G10492,train,602b46759358418dfac3d595d096fbda,b97985961e451e1e
Normal-1797.png,Normal,Normal/images/Normal-1797.png,Normal/masks/Normal-1797.png,True,G10493,train,927dba53d33495ae52cd1bfc27c5d31a,d4bf534b434f0a0b
Normal-1798.png,Normal,Normal/images/Normal-1798.png,Normal/masks/Normal-1798.png,True,G10494,val,2a1ef86e40637ec123070b55e45c7a36,d93b461c506fb8c3
Normal-1799.png,Normal,Normal/images/Normal-1799.png,Normal/masks/Normal-1799.png,True,G10495,train,b5350fb99855317965b3a7b7963e4c26,987b05061e1f4f5e
Normal-18.png,Normal,Normal/images/Normal-18.png,Normal/masks/Normal-18.png,True,G10496,test,238fa4eefd6129cb5a76652d81f873da,dc74534f544a7c4a
Normal-180.png,Normal,Normal/images/Normal-180.png,Normal/masks/Normal-180.png,True,G10497,val,9761fe8d0196317514e01da1ef5483d1,917e8e1a0d5959c7
Normal-1800.png,Normal,Normal/images/Normal-1800.png,Normal/masks/Normal-1800.png,True,G10498,val,8e1f9bb4a0dd2f0d2a6ffe418ec4b4ee,c979c257114e2f1b
Normal-1801.png,Normal,Normal/images/Normal-1801.png,Normal/masks/Normal-1801.png,True,G10499,test,d981905bf915ebfc30977f3fdb8cfe2e,9c4ad35b4c3a4b3a
Normal-1802.png,Normal,Normal/images/Normal-1802.png,Normal/masks/Normal-1802.png,True,G10500,test,f6250566ec7242eb886b0af59e72d5b5,993f861a594d1c79
Normal-1803.png,Normal,Normal/images/Normal-1803.png,Normal/masks/Normal-1803.png,True,G10501,val,f02ba28c5e65cd109f134e2ca7df6351,c93bc61a144e3bf4
Normal-1804.png,Normal,Normal/images/Normal-1804.png,Normal/masks/Normal-1804.png,True,G10502,train,fac425cd1f44e7ee5bd8fbebdb38b84e,886f071a1f567a78
Normal-1805.png,Normal,Normal/images/Normal-1805.png,Normal/masks/Normal-1805.png,True,G10503,train,5f995d021bf5271a1bf60f887c3d2e41,887947871e1647cf
Normal-1806.png,Normal,Normal/images/Normal-1806.png,Normal/masks/Normal-1806.png,True,G10504,train,2150e72903da81cf3cbbc88110402ee2,987ba6521f057b1a
Normal-1807.png,Normal,Normal/images/Normal-1807.png,Normal/masks/Normal-1807.png,True,G10505,train,441b9a24b4ddc9015cfe94fedf4b715b,d13946461c5b6f1d
Normal-1808.png,Normal,Normal/images/Normal-1808.png,Normal/masks/Normal-1808.png,True,G10506,train,5ec9db59fde3a70483bf58a2b2fe8457,c939c74b96494997
Normal-1809.png,Normal,Normal/images/Normal-1809.png,Normal/masks/Normal-1809.png,True,G10507,test,ce4cd2a5afbf85fae4d1d67ed10beae3,987dc51e5d4b1c19
Normal-181.png,Normal,Normal/images/Normal-181.png,Normal/masks/Normal-181.png,True,G10508,train,080094b15759768711afbc8faf44807d,c56f0f094d4d4ac7
Normal-1810.png,Normal,Normal/images/Normal-1810.png,Normal/masks/Normal-1810.png,True,G10509,train,1e26a7eebddda3644ed358dc65325d2d,d12e021b0f7f39ac
Normal-1811.png,Normal,Normal/images/Normal-1811.png,Normal/masks/Normal-1811.png,True,G10510,train,5659822ca856032d9ebe7d167bb578d6,c179c71b14465b5d
Normal-1812.png,Normal,Normal/images/Normal-1812.png,Normal/masks/Normal-1812.png,True,G10511,train,b1ee005303a8d9bf590a089536518679,993d865a1c4e7a1d
Normal-1813.png,Normal,Normal/images/Normal-1813.png,Normal/masks/Normal-1813.png,True,G10512,train,83d8c1efb25571bdb5079c0d094af3cb,9568865e5a5d4c79
Normal-1814.png,Normal,Normal/images/Normal-1814.png,Normal/masks/Normal-1814.png,True,G10513,train,3cbde454fcf85b1a8f446fb0d08b11d5,dd3dc71b18450a3d
Normal-1815.png,Normal,Normal/images/Normal-1815.png,Normal/masks/Normal-1815.png,True,G10514,train,83549fa8962358a1d8a18d021b47bdc7,dd2a42d70b4d0f1d
Normal-1816.png,Normal,Normal/images/Normal-1816.png,Normal/masks/Normal-1816.png,True,G10515,train,b1a44dc564bde823ab64fe35d953ef8a,c17e025b185ce9de
Normal-1817.png,Normal,Normal/images/Normal-1817.png,Normal/masks/Normal-1817.png,True,G10516,train,66da336827a579a12a352eedd383b125,9d2ec61f1a4d0e3a
Normal-1818.png,Normal,Normal/images/Normal-1818.png,Normal/masks/Normal-1818.png,True,G10517,val,ebc6fffd612382672726b7a663163927,d9e88611878e7ac7
Normal-1819.png,Normal,Normal/images/Normal-1819.png,Normal/masks/Normal-1819.png,True,G10518,train,734c938f83e4de0704b5e8d6e189b448,f969471e1c4c1b1d
Normal-182.png,Normal,Normal/images/Normal-182.png,Normal/masks/Normal-182.png,True,G10519,val,05336d6c432d63467d97f29a62bfea35,812a861ae8f979d7
Normal-1820.png,Normal,Normal/images/Normal-1820.png,Normal/masks/Normal-1820.png,True,G10520,test,aca9771b37cf4db1d12b6dc5041ea2a5,d03d4b431d0b8fad
Normal-1821.png,Normal,Normal/images/Normal-1821.png,Normal/masks/Normal-1821.png,True,G10521,test,b2c1204e031fcfec2135b54db23abd89,8139861e1f471f3d
Normal-1822.png,Normal,Normal/images/Normal-1822.png,Normal/masks/Normal-1822.png,True,G10522,train,a07b34c95f05ec1b335efca7554944f4,897ad61b423f382d
Normal-1823.png,Normal,Normal/images/Normal-1823.png,Normal/masks/Normal-1823.png,True,G10523,train,60a155985e1dd5daa24fb71389e18245,dd2c574b5d4d4c0a
Normal-1824.png,Normal,Normal/images/Normal-1824.png,Normal/masks/Normal-1824.png,True,G10524,train,66f56ae1edc197daafc8ae0c394b250e,807b07861e1d7b5b
Normal-1825.png,Normal,Normal/images/Normal-1825.png,Normal/masks/Normal-1825.png,True,G10525,train,be7ffe9c4ac12ebd2f6e333824f69c2e,c83dd25bda4b5a42
Normal-1826.png,Normal,Normal/images/Normal-1826.png,Normal/masks/Normal-1826.png,True,G10526,train,65d5cd7c95c573c7b6fed383e69cbe27,d17bc61a194b1d1e
Normal-1827.png,Normal,Normal/images/Normal-1827.png,Normal/masks/Normal-1827.png,True,G10527,train,03f2f6aa8fb2c9f383a3610c1240cf47,dc0cc20b0dafa9bd
Normal-1828.png,Normal,Normal/images/Normal-1828.png,Normal/masks/Normal-1828.png,True,G10528,train,fdf5a484c1a5588e24d230933e1c67e9,dc3cc35f504e0d3c
Normal-1829.png,Normal,Normal/images/Normal-1829.png,Normal/masks/Normal-1829.png,True,G10529,train,c3d437b9bc653dd8f656eddd53afaeb7,dc3cc34b19453c3d
Normal-183.png,Normal,Normal/images/Normal-183.png,Normal/masks/Normal-183.png,True,G10530,train,695c9459a7357fe2b407775430bfbab3,d12bc23bfc693960
Normal-1830.png,Normal,Normal/images/Normal-1830.png,Normal/masks/Normal-1830.png,True,G10531,train,c947564c78983d8633c63bbc1974c74b,89edc75a15c61ac5
Normal-1831.png,Normal,Normal/images/Normal-1831.png,Normal/masks/Normal-1831.png,True,G10532,train,0223958ff51fcafa90bc3affc60df39c,d92f061e185c6b79
Normal-1832.png,Normal,Normal/images/Normal-1832.png,Normal/masks/Normal-1832.png,True,G10533,train,13bd1cec2bb682c87929f7b125ef5305,9939d61b1cca1b39
Normal-1833.png,Normal,Normal/images/Normal-1833.png,Normal/masks/Normal-1833.png,True,G10534,train,7ea93c8530a8535c9a732b614702b539,d979c65a1f491b18
Normal-1834.png,Normal,Normal/images/Normal-1834.png,Normal/masks/Normal-1834.png,True,G10535,train,2178a5dd0fa217d8d02379cc10cd0407,d93b071e1849e9bc
Normal-1835.png,Normal,Normal/images/Normal-1835.png,Normal/masks/Normal-1835.png,True,G10536,train,0fa3ca598603ff3fb0b2a2462df31078,9969071a5f4d1a5d
Normal-1836.png,Normal,Normal/images/Normal-1836.png,Normal/masks/Normal-1836.png,True,G10537,train,4650c08b3cd3cca5585c0b09afcee5a4,c93f061a19cda9b9
Normal-1837.png,Normal,Normal/images/Normal-1837.png,Normal/masks/Normal-1837.png,True,G10538,val,7e612796cdace7c394352469d0c62563,b92d061e594d785d
Normal-1838.png,Normal,Normal/images/Normal-1838.png,Normal/masks/Normal-1838.png,True,G10539,val,85916814892efc548fd6f7b89050446b,982f061b0d38ed7b
Normal-1839.png,Normal,Normal/images/Normal-1839.png,Normal/masks/Normal-1839.png,True,G10540,val,a54e9db1822074c7cb653c7dc2b16828,897807871e1fc65e
Normal-184.png,Normal,Normal/images/Normal-184.png,Normal/masks/Normal-184.png,True,G10541,test,6cc84f5af99a14d2db4c9dd117ba0940,d47e871f8d0c1ccc
Normal-1840.png,Normal,Normal/images/Normal-1840.png,Normal/masks/Normal-1840.png,True,G10542,test,db394fd6fb2d6b11f760b045698a98fd,dd67420f4b0b0e2f
Normal-1841.png,Normal,Normal/images/Normal-1841.png,Normal/masks/Normal-1841.png,True,G10543,test,3d026c6f3db777c48106cb81b0120c55,d979851e841e934f
Normal-1842.png,Normal,Normal/images/Normal-1842.png,Normal/masks/Normal-1842.png,True,G10544,train,680d195b2e005936c58e87a7590e18ca,99798605185efe59
Normal-1843.png,Normal,Normal/images/Normal-1843.png,Normal/masks/Normal-1843.png,True,G10545,train,2fcafd33d905866fcbb452ddc4f7932a,d42f464d0a2cedbc
Normal-1844.png,Normal,Normal/images/Normal-1844.png,Normal/masks/Normal-1844.png,True,G10546,train,f77d57c6677002837b6fb028a891c70f,dd3dc2430b934b0f
Normal-1845.png,Normal,Normal/images/Normal-1845.png,Normal/masks/Normal-1845.png,True,G10547,train,32d0afc3f8c230a16ce18d5daa21f8dd,913a465b196c7b39
Normal-1846.png,Normal,Normal/images/Normal-1846.png,Normal/masks/Normal-1846.png,True,G10548,train,6a94d9dd9541c0ea4556d88a1658445d,dc2ec3435d4a4a5e
Normal-1847.png,Normal,Normal/images/Normal-1847.png,Normal/masks/Normal-1847.png,True,G10549,train,552c9b535b44c38cf8d269b755201b9f,997985161a1d5f5a
Normal-1848.png,Normal,Normal/images/Normal-1848.png,Normal/masks/Normal-1848.png,True,G10550,train,45882af4710cd0c7dc36b1cade3d5ef8,dd3cc21b464b1c4f
Normal-1849.png,Normal,Normal/images/Normal-1849.png,Normal/masks/Normal-1849.png,True,G10551,train,18891299983bd86a7469fed925c69b74,887f85961e095f1e
Normal-185.png,Normal,Normal/images/Normal-185.png,Normal/masks/Normal-185.png,True,G10552,train,1f28d788067e869835e4f14d41d47469,9125c63af8693d65
Normal-1850.png,Normal,Normal/images/Normal-1850.png,Normal/masks/Normal-1850.png,True,G10553,train,c58929ca50765a51249c499273d1c5d6,9169c61e165e3a79
Normal-1851.png,Normal,Normal/images/Normal-1851.png,Normal/masks/Normal-1851.png,True,G10554,test,a3e90ab74ec3724f4803c326f8761424,c03c4b4b183e6f3d
Normal-1852.png,Normal,Normal/images/Normal-1852.png,Normal/masks/Normal-1852.png,True,G10555,train,a4fbaa706eddacfdb961b1d1a82f961e,dd28d30b0e2c3fb4
Normal-1853.png,Normal,Normal/images/Normal-1853.png,Normal/masks/Normal-1853.png,True,G10556,train,8917af7e50b30b5d4fc39107d8251c64,817906861a9f7ed9
Normal-1854.png,Normal,Normal/images/Normal-1854.png,Normal/masks/Normal-1854.png,True,G10557,val,4ff9554e679246c8c2fb1ad467313293,d42f420f2cecf5d0
Normal-1855.png,Normal,Normal/images/Normal-1855.png,Normal/masks/Normal-1855.png,True,G10558,val,5315c1f745e497e743f68e58a25e2fdb,d43d03822d3e6d79
Normal-1856.png,Normal,Normal/images/Normal-1856.png,Normal/masks/Normal-1856.png,True,G10559,val,2677d43c86a7fdfb907dacf5d7101f0e,9977161a181b3b3e
Normal-1857.png,Normal,Normal/images/Normal-1857.png,Normal/masks/Normal-1857.png,True,G10560,test,f68efda165ed652d7253c11ad3a96cc9,dc2cc2d34b4dce43
Normal-1858.png,Normal,Normal/images/Normal-1858.png,Normal/masks/Normal-1858.png,True,G10561,test,ca3308acdc21b7aa8bd6ae925ac4644b,9978065f1e465bb8
Normal-1859.png,Normal,Normal/images/Normal-1859.png,Normal/masks/Normal-1859.png,True,G10562,test,29892e2749bf03a4175e8a8612867f12,993c461b444f1f4f
Normal-186.png,Normal,Normal/images/Normal-186.png,Normal/masks/Normal-186.png,True,G10563,train,6d183b0454ed92f820ee0bc84e12b9fc,9135c63aac4f3965
Normal-1860.png,Normal,Normal/images/Normal-1860.png,Normal/masks/Normal-1860.png,True,G10564,train,f4acc7c4454565b7e97dd71db6fcef39,992d061ac45af8ef
Normal-1861.png,Normal,Normal/images/Normal-1861.png,Normal/masks/Normal-1861.png,True,G10565,train,b67efa6f784f3009e6fe2148645e5ae2,997e061a191b1f7a
Normal-1862.png,Normal,Normal/images/Normal-1862.png,Normal/masks/Normal-1862.png,True,G10566,train,cfc23c6c0501cf57353aa40f51a96299,d53f42970a956a9c
Normal-1863.png,Normal,Normal/images/Normal-1863.png,Normal/masks/Normal-1863.png,True,G10567,train,80c6f64213c97812251aa3ccf389f45a,817956c7181ec77a
Normal-1864.png,Normal,Normal/images/Normal-1864.png,Normal/masks/Normal-1864.png,True,G10568,train,06059392b1b1b6595173bf382f8575f4,dd2e40535e445e1f
Normal-1865.png,Normal,Normal/images/Normal-1865.png,Normal/masks/Normal-1865.png,True,G10569,train,2e9088ac3d564337798155c7a0b6bc31,9939c71a184b3b79
Normal-1866.png,Normal,Normal/images/Normal-1866.png,Normal/masks/Normal-1866.png,True,G10570,train,70be8c9c7fa2257d1e066c9a9faf6896,d43d42d3d94c4c3d
Normal-1867.png,Normal,Normal/images/Normal-1867.png,Normal/masks/Normal-1867.png,True,G10571,val,0c81c2d8d41335093fa60b5c1aa4f5ca,8178571718175e7b
Normal-1868.png,Normal,Normal/images/Normal-1868.png,Normal/masks/Normal-1868.png,True,G10572,val,7cd903af1575f7dc7176ca1e0a329ba3,c97bc65a194d4b38
Normal-1869.png,Normal,Normal/images/Normal-1869.png,Normal/masks/Normal-1869.png,True,G10573,train,5a816c308ca10aed8ed3cc57a0c9476d,c43dc3c51aca8f9a
Normal-187.png,Normal,Normal/images/Normal-187.png,Normal/masks/Normal-187.png,True,G10574,train,68e3033499491183397b49c0c089ea7b,802f8719cd6bfac2
Normal-1870.png,Normal,Normal/images/Normal-1870.png,Normal/masks/Normal-1870.png,True,G10575,train,f67c0b0994cb280b365b4f2de429cdf0,d13d42165e4e3b1b
Normal-1871.png,Normal,Normal/images/Normal-1871.png,Normal/masks/Normal-1871.png,True,G10576,train,1c78e320ff8dd101621783144544c9e6,99695e5f587831e0
Normal-1872.png,Normal,Normal/images/Normal-1872.png,Normal/masks/Normal-1872.png,True,G10577,test,f95cffeae21a75f664ecf3012cf47e47,997f461a190d3f19
Normal-1873.png,Normal,Normal/images/Normal-1873.png,Normal/masks/Normal-1873.png,True,G10578,test,6996a24f25b187bc857c0f7646a49a4b,d97d071b055a39e8
Normal-1874.png,Normal,Normal/images/Normal-1874.png,Normal/masks/Normal-1874.png,True,G10579,train,b0cff79a5daabd08bd16884e661e519c,d03cc6bf390f2c38
Normal-1875.png,Normal,Normal/images/Normal-1875.png,Normal/masks/Normal-1875.png,True,G10580,train,98117a65784cf9e469bf1290e09d7689,997b571a181939b9
Normal-1876.png,Normal,Normal/images/Normal-1876.png,Normal/masks/Normal-1876.png,True,G10581,val,2c2a5e389755b14271035ff8b66a631e,d93b468e5a046f1d
Normal-1877.png,Normal,Normal/images/Normal-1877.png,Normal/masks/Normal-1877.png,True,G10582,test,262488c281287f4d28b4d72e716e5f8a,986d561e475a195b
Normal-1878.png,Normal,Normal/images/Normal-1878.png,Normal/masks/Normal-1878.png,True,G10583,train,1f783e78d3cc57c5a7d9032eae7fbdc0,dd2ec21f096e0bf0
Normal-1879.png,Normal,Normal/images/Normal-1879.png,Normal/masks/Normal-1879.png,True,G10584,train,94482496169098ab07c4392df549afda,d4bc2fc30958c26f
Normal-188.png,Normal,Normal/images/Normal-188.png,Normal/masks/Normal-188.png,True,G10585,train,0baf0daed7d1830d86e1369f8e6c39ba,d07cc73d8d0d1ccc
Normal-1880.png,Normal,Normal/images/Normal-1880.png,Normal/masks/Normal-1880.png,True,G10586,train,58d71ba07f5eb1cc9769f6f473ac3c33,d92c071a074f3e4f
Normal-1881.png,Normal,Normal/images/Normal-1881.png,Normal/masks/Normal-1881.png,True,G10587,test,62edfbbcc124cabc9671266f73468001,dc3dd20f1e420f5c
Normal-1882.png,Normal,Normal/images/Normal-1882.png,Normal/masks/Normal-1882.png,True,G10588,train,9697b4a6376214ecc206ab19f1e7e8b3,8ab4b5cbf072270e
Normal-1883.png,Normal,Normal/images/Normal-1883.png,Normal/masks/Normal-1883.png,True,G10589,train,dc921cd1c3eddd3913b8d0ac75f1cf47,d42ad0534b5b5c4f
Normal-1884.png,Normal,Normal/images/Normal-1884.png,Normal/masks/Normal-1884.png,True,G10590,train,ef821981d2be509fd43e1dc6011b81d9,9978851a1d5e7ac5
Normal-1885.png,Normal,Normal/images/Normal-1885.png,Normal/masks/Normal-1885.png,True,G10591,train,a81c97619a7708c0cb8a30e780985600,9c28c713cfc31e17
Normal-1886.png,Normal,Normal/images/Normal-1886.png,Normal/masks/Normal-1886.png,True,G10592,train,bb03f57d41616d2270993e8c32745fbf,887e475f19456db0
Normal-1887.png,Normal,Normal/images/Normal-1887.png,Normal/masks/Normal-1887.png,True,G10593,train,12c18ef6fa0c1477164ab5c35205d102,d918131b183bbfb4
Normal-1888.png,Normal,Normal/images/Normal-1888.png,Normal/masks/Normal-1888.png,True,G10594,train,23964d1250ea85d07d4f9a4575a9df6a,d42d68e0d593ba55
Normal-1889.png,Normal,Normal/images/Normal-1889.png,Normal/masks/Normal-1889.png,True,G10595,train,6aceb1c8df1f3cd76328d6f5347480dc,a969861e5959594f
Normal-189.png,Normal,Normal/images/Normal-189.png,Normal/masks/Normal-189.png,True,G10596,train,4ba4765f8ebb2b63daa8c324b96b07c8,d07cc73b9d0d3c84
Normal-1890.png,Normal,Normal/images/Normal-1890.png,Normal/masks/Normal-1890.png,True,G10597,train,706b75fdf20caf6754157ac329ea6ac6,d939861b144d3b6d
Normal-1891.png,Normal,Normal/images/Normal-1891.png,Normal/masks/Normal-1891.png,True,G10598,val,508f1c6288a0338a102c0f5f2825476d,dc2cd2d14f0aad0f
Normal-1892.png,Normal,Normal/images/Normal-1892.png,Normal/masks/Normal-1892.png,True,G10599,val,00f45c42705ea083c58717ac0216c035,81797ec6521b195e
Normal-1893.png,Normal,Normal/images/Normal-1893.png,Normal/masks/Normal-1893.png,True,G10600,val,cfa844826dbdec3b9c111400433955cb,997f464a1b434f1c
Normal-1894.png,Normal,Normal/images/Normal-1894.png,Normal/masks/Normal-1894.png,True,G10601,test,8e6f6f61218da073db7ae4699073f810,8979c78e1a854a3f
Normal-1895.png,Normal,Normal/images/Normal-1895.png,Normal/masks/Normal-1895.png,True,G10602,test,1e893976efcd4bb95f0c1af1597d6f76,9978061a8f5e38ed
Normal-1896.png,Normal,Normal/images/Normal-1896.png,Normal/masks/Normal-1896.png,True,G10603,train,60c3d3bbaacf5fb5c5f4bbb5338f7587,9979c61a19cb4cba
Normal-1897.png,Normal,Normal/images/Normal-1897.png,Normal/masks/Normal-1897.png,True,G10604,train,ba10233d3f96fc07eb627140bf1d2e78,885b07161ec97bb9
Normal-1898.png,Normal,Normal/images/Normal-1898.png,Normal/masks/Normal-1898.png,True,G10605,train,760ebeb2bd893909db4ecb4ea3b59524,d979c75213c55a1a
Normal-1899.png,Normal,Normal/images/Normal-1899.png,Normal/masks/Normal-1899.png,True,G10606,train,f1dab16fd62e785b87f573ebbc5b8003,91695e1e1979e968
Normal-19.png,Normal,Normal/images/Normal-19.png,Normal/masks/Normal-19.png,True,G10607,train,f90c31b1d266fedc41e8fd1a299d3016,c839c21f1c4e39bd
Normal-190.png,Normal,Normal/images/Normal-190.png,Normal/masks/Normal-190.png,True,G10608,train,02a49e34c88a6a6cb0b1ba7e8a495c7d,9139c61a9ada39b5
Normal-1900.png,Normal,Normal/images/Normal-1900.png,Normal/masks/Normal-1900.png,True,G10609,train,9d5fddacb787a18231e5eb58e7954403,d9bdc21b05451f69
Normal-1901.png,Normal,Normal/images/Normal-1901.png,Normal/masks/Normal-1901.png,True,G10610,val,ae7b0ab7aa0ae8fd993c7c4f931205ce,8179161e7a5de839
Normal-1902.png,Normal,Normal/images/Normal-1902.png,Normal/masks/Normal-1902.png,True,G10611,train,173008088ed819b6ef8635f13a9b47ca,dd2cc61b051f1d4d
Normal-1903.png,Normal,Normal/images/Normal-1903.png,Normal/masks/Normal-1903.png,True,G10612,test,e40193b54972bb777070a533d5773f72,d47d074f0d0c2f2d
Normal-1904.png,Normal,Normal/images/Normal-1904.png,Normal/masks/Normal-1904.png,True,G10613,test,cb1b5cf2ad6cdb66e1ef23012014e169,887d461b1f4e39e4
Normal-1905.png,Normal,Normal/images/Normal-1905.png,Normal/masks/Normal-1905.png,True,G10614,train,9ff309050154aa447bee74397b7a7346,9979d61a451b394b
Normal-1906.png,Normal,Normal/images/Normal-1906.png,Normal/masks/Normal-1906.png,True,G10615,train,7a0468d81ed279f9fe1ae0bfca522560,dd2ad35f5c0a1d22
Normal-1907.png,Normal,Normal/images/Normal-1907.png,Normal/masks/Normal-1907.png,True,G10616,train,257b797c521c51f758e21f1a93bfce98,dd2fc21d524e1b4a
Normal-1908.png,Normal,Normal/images/Normal-1908.png,Normal/masks/Normal-1908.png,True,G10617,val,d258b387788216787f8ff1a66656ba00,9168c7161f496f3c
Normal-1909.png,Normal,Normal/images/Normal-1909.png,Normal/masks/Normal-1909.png,True,G10618,train,541279cb857d02c976ab6d9df5e5be63,807a47871e1f4e1f
Normal-191.png,Normal,Normal/images/Normal-191.png,Normal/masks/Normal-191.png,True,G10619,train,74265e47062c2f58767dc3118c42d059,d03ec31bed4538c6
Normal-1910.png,Normal,Normal/images/Normal-1910.png,Normal/masks/Normal-1910.png,True,G10620,train,4b02ea7463363d727f0daefe4f2f8bc8,8979071a195f5de1
Normal-1911.png,Normal,Normal/images/Normal-1911.png,Normal/masks/Normal-1911.png,True,G10621,train,75f21d021233d5821817462b13403f89,dc2ac34f0c4f0ecb
Normal-1912.png,Normal,Normal/images/Normal-1912.png,Normal/masks/Normal-1912.png,True,G10622,val,c7ff94b31ae16a3c346a6f1d24989b1b,dd3d42575b404b1b
Normal-1913.png,Normal,Normal/images/Normal-1913.png,Normal/masks/Normal-1913.png,True,G10623,test,f442d7f0db4829fda919f95058f247c4,cc3d43520d0dfd1d
Normal-1914.png,Normal,Normal/images/Normal-1914.png,Normal/masks/Normal-1914.png,True,G10624,train,c9e135db38351d82213fd08c85064376,d52ec34b4d4d094f
Normal-1915.png,Normal,Normal/images/Normal-1915.png,Normal/masks/Normal-1915.png,True,G10625,train,72ab6e84df67837e8991fc148e7deb43,d42e43cb0b4f0e1f
Normal-1916.png,Normal,Normal/images/Normal-1916.png,Normal/masks/Normal-1916.png,True,G10626,train,b417645835c9478cc785e8dde7b4a25b,d92d061a164b3f6b
Normal-1917.png,Normal,Normal/images/Normal-1917.png,Normal/masks/Normal-1917.png,True,G10627,train,86de1522a4027318c7a9d68ff7726f7b,997d461e691c698e
Normal-1918.png,Normal,Normal/images/Normal-1918.png,Normal/masks/Normal-1918.png,True,G10628,train,fd39147f44f0627067848fcf7ea86e9e,dc2d42530f4a2f3d
Normal-1919.png,Normal,Normal/images/Normal-1919.png,Normal/masks/Normal-1919.png,True,G10629,train,88d6c70343eb4d03da0c5e42f0ff8a0b,997dc61a59451e78
Normal-192.png,Normal,Normal/images/Normal-192.png,Normal/masks/Normal-192.png,True,G10630,train,3b632637df81c5ba2f8e2949aef20d82,d47ec70f8d4c2cc4
Normal-1920.png,Normal,Normal/images/Normal-1920.png,Normal/masks/Normal-1920.png,True,G10631,train,5a737d8888f9df868e92b9c2a7018f10,dd2a544b1b4d0a3f
Normal-1921.png,Normal,Normal/images/Normal-1921.png,Normal/masks/Normal-1921.png,True,G10632,train,ac9bb101bab17162e5daa0164e55868c,d93e461b4459d94e
Normal-1922.png,Normal,Normal/images/Normal-1922.png,Normal/masks/Normal-1922.png,True,G10633,train,e470c4c4939d1dcf5503a0753e9c4e1d,dd2f425f444f1c49
Normal-1923.png,Normal,Normal/images/Normal-1923.png,Normal/masks/Normal-1923.png,True,G10634,train,ff08ea1daeb550eeaef03d936e3d52b9,807f861e194f5d5c
Normal-1924.png,Normal,Normal/images/Normal-1924.png,Normal/masks/Normal-1924.png,True,G10635,train,837d2ce3b795adeeece78b42d5c020b8,dc2dc25b4d460f6c
Normal-1925.png,Normal,Normal/images/Normal-1925.png,Normal/masks/Normal-1925.png,True,G10636,train,94468ed6b2581c366a5a3e899415b260,993d460b1b0d6f39
Normal-1926.png,Normal,Normal/images/Normal-1926.png,Normal/masks/Normal-1926.png,True,G10637,val,3acf6b2b890078ccd9d4c4de005fed31,dd2cd24b51430f5d
Normal-1927.png,Normal,Normal/images/Normal-1927.png,Normal/masks/Normal-1927.png,True,G10638,val,f364e56b0fa78108929c5f54b46d3261,c17fc24d7841633b
Normal-1928.png,Normal,Normal/images/Normal-1928.png,Normal/masks/Normal-1928.png,True,G10639,val,5533f01dea2c666400a25b50cacb1416,dc2d424b0c2cedbd
Normal-1929.png,Normal,Normal/images/Normal-1929.png,Normal/masks/Normal-1929.png,True,G10640,val,cb4f0964d19613b56bdec07e07ec961c,c838d2531bc75f19
Normal-193.png,Normal,Normal/images/Normal-193.png,Normal/masks/Normal-193.png,True,G10641,test,8892478a5aa7a5756ba6063123ec3619,d47ea31fcd2c0cc4
Normal-1930.png,Normal,Normal/images/Normal-1930.png,Normal/masks/Normal-1930.png,True,G10642,test,6bc5627b29244b9fabc2b4f102c8c3a0,dd7d06521b016f1e
Normal-1931.png,Normal,Normal/images/Normal-1931.png,Normal/masks/Normal-1931.png,True,G10643,test,634d7bf704604fa9481775c7b016cc44,c83d50535f52435f
Normal-1932.png,Normal,Normal/images/Normal-1932.png,Normal/masks/Normal-1932.png,True,G10644,train,4a53a7f7924095dbcfbc98576e3525b1,952c075c7d68eae2
Normal-1933.png,Normal,Normal/images/Normal-1933.png,Normal/masks/Normal-1933.png,True,G10645,train,55583d3c61a94509cfda2f55ef855d9a,8178169717975e9a
Normal-1934.png,Normal,Normal/images/Normal-1934.png,Normal/masks/Normal-1934.png,True,G10646,train,1501f4ea61cc853d32ea7c5d119af107,dd2ad0531f4c0b1f
Normal-1935.png,Normal,Normal/images/Normal-1935.png,Normal/masks/Normal-1935.png,True,G10647,train,7adf4ba346a7876415401a5c4f8d1039,c93a56d71ec93a90
Normal-1936.png,Normal,Normal/images/Normal-1936.png,Normal/masks/Normal-1936.png,True,G10648,train,de9b4da5f2cf9a4a8debb91563faae46,915b841c5a797a7a
Normal-1937.png,Normal,Normal/images/Normal-1937.png,Normal/masks/Normal-1937.png,True,G10649,val,8d3bc59e72e62715de12b7e3a7228f36,d43cc34b084e6f3d
Normal-1938.png,Normal,Normal/images/Normal-1938.png,Normal/masks/Normal-1938.png,True,G10650,train,08f3444fd99a9734a3e40d68db51c13a,dd7d02521f4e3878
Normal-1939.png,Normal,Normal/images/Normal-1939.png,Normal/masks/Normal-1939.png,True,G10651,train,5147967da7de4e03f2c1eee9d6fa6c33,c93d061b594c78f9
Normal-194.png,Normal,Normal/images/Normal-194.png,Normal/masks/Normal-194.png,True,G10652,train,60dabea97194d35503a8f437090ee3d3,90edc73ac96d384c
Normal-1940.png,Normal,Normal/images/Normal-1940.png,Normal/masks/Normal-1940.png,True,G10653,train,06f9e83ec9e048746f2884b29ae2a23d,c42e3fd1708bcd2c
Normal-1941.png,Normal,Normal/images/Normal-1941.png,Normal/masks/Normal-1941.png,True,G10654,train,8c8a56b41b306775c907ac535435f71e,dc7c43430b4f4f2a
Normal-1942.png,Normal,Normal/images/Normal-1942.png,Normal/masks/Normal-1942.png,True,G10655,test,bfaaaa296d54f37a96916ef22eacfc34,d929c25f524b394b
Normal-1943.png,Normal,Normal/images/Normal-1943.png,Normal/masks/Normal-1943.png,True,G10656,train,2479edacf9fd1944d4648539725a986e,917a16181beff885
Normal-1944.png,Normal,Normal/images/Normal-1944.png,Normal/masks/Normal-1944.png,True,G10657,test,9ef9c4fc3f18279cadeb57b909cc3aa4,f979c65e1c465918
Normal-1945.png,Normal,Normal/images/Normal-1945.png,Normal/masks/Normal-1945.png,True,G10658,train,18ce280fc8357fae890dddf040d0f29e,9d680ecfd8cc8d85
Normal-1946.png,Normal,Normal/images/Normal-1946.png,Normal/masks/Normal-1946.png,True,G10659,train,2f2408c3bc80159f0b2c4e494f40b7cb,d02fd2cf5c04693d
Normal-1947.png,Normal,Normal/images/Normal-1947.png,Normal/masks/Normal-1947.png,True,G10660,train,5079ab5f7f3795b81ddcb6eae4268561,9d2e865f0d4c4ccd
Normal-1948.png,Normal,Normal/images/Normal-1948.png,Normal/masks/Normal-1948.png,True,G10661,val,d60a8838f9b5c44bf3714146447d3a7b,d139d35b1a0d3939
Normal-1949.png,Normal,Normal/images/Normal-1949.png,Normal/masks/Normal-1949.png,True,G10662,val,f1f8ded9e43f7423490e5684671bb65e,dc3c434919ae2dbc
Normal-195.png,Normal,Normal/images/Normal-195.png,Normal/masks/Normal-195.png,True,G10663,test,60374cfc97cb558a5a8fdd323d076056,913ce61ae8cd6995
Normal-1950.png,Normal,Normal/images/Normal-1950.png,Normal/masks/Normal-1950.png,True,G10664,train,816960e789074be3ab2005dcc239b565,997907061e1fc55e
Normal-1951.png,Normal,Normal/images/Normal-1951.png,Normal/masks/Normal-1951.png,True,G10665,train,5659526c3a4ff704e73e51d54c7ce444,99590e9a1859ecfa
Normal-1952.png,Normal,Normal/images/Normal-1952.png,Normal/masks/Normal-1952.png,True,G10666,train,fccc6c7c1daacba40197cea4fbaf5b55,9161ee1eb8790ce9
Normal-1953.png,Normal,Normal/images/Normal-1953.png,Normal/masks/Normal-1953.png,True,G10667,train,0adcfbfeccf02509e5e0014da7c9dd11,b805835b4d5c6c7e
Normal-1954.png,Normal,Normal/images/Normal-1954.png,Normal/masks/Normal-1954.png,True,G10668,train,07d316178adfd1a56807fc06fe111741,817905c6165e5f4f
Normal-1955.png,Normal,Normal/images/Normal-1955.png,Normal/masks/Normal-1955.png,True,G10669,train,a68bd0b10b4a78644db429f95e019307,fd72d30f441c5c46
Normal-1956.png,Normal,Normal/images/Normal-1956.png,Normal/masks/Normal-1956.png,True,G10670,train,ce5d1ed214a4f7c02adf110dbc4a7923,dd6d061a19475a79
Normal-1957.png,Normal,Normal/images/Normal-1957.png,Normal/masks/Normal-1957.png,True,G10671,train,3557a4c2b5697d0f7097c8f81bd80124,dd2ad20b4a0a6f2f
Normal-1958.png,Normal,Normal/images/Normal-1958.png,Normal/masks/Normal-1958.png,True,G10672,train,bfc0766593dceebc89c8566cae339038,91791e19199e79b1
Normal-1959.png,Normal,Normal/images/Normal-1959.png,Normal/masks/Normal-1959.png,True,G10673,train,db190cb55c34a5defe4e1f61e0ba0c4d,dd7cc25a4c4d584d
Normal-196.png,Normal,Normal/images/Normal-196.png,Normal/masks/Normal-196.png,True,G10674,train,4c4b321bbde097865959766e5de9b82f,d462c779c56ce0c6
Normal-1960.png,Normal,Normal/images/Normal-1960.png,Normal/masks/Normal-1960.png,True,G10675,train,e780ca648c7ae20c07e85c297d80ecb4,dd2cd24b1f544a1d
Normal-1961.png,Normal,Normal/images/Normal-1961.png,Normal/masks/Normal-1961.png,True,G10676,train,d4946259c1bc67418ea139efb284910c,901f165a59fa78e8
Normal-1962.png,Normal,Normal/images/Normal-1962.png,Normal/masks/Normal-1962.png,True,G10677,train,96e01eb3ebaf7d11367fc139060eb008,dc2e434b0f0a2f3d
Normal-1963.png,Normal,Normal/images/Normal-1963.png,Normal/masks/Normal-1963.png,True,G10678,val,d64d90e41e34fca2a02ac5383f842c55,d1f986160e961a9f
Normal-1964.png,Normal,Normal/images/Normal-1964.png,Normal/masks/Normal-1964.png,True,G10679,val,e8794a4c541bce8fcb58f1520e6c417a,8807175e5c796d78
Normal-1965.png,Normal,Normal/images/Normal-1965.png,Normal/masks/Normal-1965.png,True,G10680,val,e5429d728d9758d2c1589f96b1605aad,c03f021f1f4c2f5d
Normal-1966.png,Normal,Normal/images/Normal-1966.png,Normal/masks/Normal-1966.png,True,G10681,test,0f2009ab907d437f1d45fc6080bc4b58,997dc2d31b4f4458
Normal-1967.png,Normal,Normal/images/Normal-1967.png,Normal/masks/Normal-1967.png,True,G10682,test,a4f0ea5863ce78b5745fbce91c4b5cc7,8979070a5f556ab9
Normal-1968.png,Normal,Normal/images/Normal-1968.png,Normal/masks/Normal-1968.png,True,G10683,test,a3c37c80f03a2de792385b867c80fa82,993e4659594e38ad
Normal-1969.png,Normal,Normal/images/Normal-1969.png,Normal/masks/Normal-1969.png,True,G10684,train,54ef672b646fe14db29b6feeebe5548d,dc2cc30b0d4e3fe8
Normal-197.png,Normal,Normal/images/Normal-197.png,Normal/masks/Normal-197.png,True,G10685,train,75dcf0ce30efb18d5aa2d8e1a586a5e1,813bce1af84939d9
Normal-1970.png,Normal,Normal/images/Normal-1970.png,Normal/masks/Normal-1970.png,True,G10686,train,67313a39ca349d571ca082ce569e1f17,993dc70f1a450f39
Normal-1971.png,Normal,Normal/images/Normal-1971.png,Normal/masks/Normal-1971.png,True,G10687,train,93d7c97cd88f9919f47d790d73b54671,d93f43164b1d491b
Normal-1972.png,Normal,Normal/images/Normal-1972.png,Normal/masks/Normal-1972.png,True,G10688,test,b2c1dcde6cda3854dfbdb4c3b6d968cf,997a87161b055e1f
Normal-1973.png,Normal,Normal/images/Normal-1973.png,Normal/masks/Normal-1973.png,True,G10689,test,b58b6dbeb6a705a9f860590397cea894,dc2c420b4b4d6e7e
Normal-1974.png,Normal,Normal/images/Normal-1974.png,Normal/masks/Normal-1974.png,True,G10690,train,688b13a9182dc348338d64c7cb8c48b4,992e461b865b1e4f
Normal-1975.png,Normal,Normal/images/Normal-1975.png,Normal/masks/Normal-1975.png,True,G10691,train,0059289591a392d325962d8c2e7fb983,dc2dc2431f4aee38
Normal-1976.png,Normal,Normal/images/Normal-1976.png,Normal/masks/Normal-1976.png,True,G10692,train,ef7edb1cf395260729b5017ebafc5ed9,8979c6861e0f4f1b
Normal-1977.png,Normal,Normal/images/Normal-1977.png,Normal/masks/Normal-1977.png,True,G10693,train,b13e7cc09c265f4d68d3fb1bb69adf2d,bc3c435b4b4f0a4e
Normal-1978.png,Normal,Normal/images/Normal-1978.png,Normal/masks/Normal-1978.png,True,G10694,train,1b6df3a1b3c10b4560c9f7a6d913fc88,83780c0d1e5e5b7b
Normal-1979.png,Normal,Normal/images/Normal-1979.png,Normal/masks/Normal-1979.png,True,G10695,val,73dae3b839a7d0434b6c32b6210246cf,c979d6861a54d61b
Normal-198.png,Normal,Normal/images/Normal-198.png,Normal/masks/Normal-198.png,True,G10696,train,3b523c224d472364c82887f39f0e3081,c5cac25aecd178d4
Normal-1980.png,Normal,Normal/images/Normal-1980.png,Normal/masks/Normal-1980.png,True,G10697,train,216de78828aaac665607cffb83677713,d43c03931f4b4d3e
Normal-1981.png,Normal,Normal/images/Normal-1981.png,Normal/masks/Normal-1981.png,True,G10698,train,d0600d464fcd708be9c9c9555c89f78a,d12b421b192b3fad
Normal-1982.png,Normal,Normal/images/Normal-1982.png,Normal/masks/Normal-1982.png,True,G10699,train,ab67b2a620ad6b2404af8d5e43987219,9d2cc213574f1acb
Normal-1983.png,Normal,Normal/images/Normal-1983.png,Normal/masks/Normal-1983.png,True,G10700,train,d4d843a411e301affc133236d47565bc,8169161a5e1f5b1f
Normal-1984.png,Normal,Normal/images/Normal-1984.png,Normal/masks/Normal-1984.png,True,G10701,train,da1065361b1219a8864c3e6abf10cce4,dd2f421b460f1a4f
Normal-1985.png,Normal,Normal/images/Normal-1985.png,Normal/masks/Normal-1985.png,True,G10702,val,3f998adcb483ee955adcbc79048f8796,d16c4a190e2dbdad
Normal-1986.png,Normal,Normal/images/Normal-1986.png,Normal/masks/Normal-1986.png,True,G10703,val,6364a6bc67af5ec679542a01268c45af,997f861a0c4d78ad
Normal-1987.png,Normal,Normal/images/Normal-1987.png,Normal/masks/Normal-1987.png,True,G10704,test,f30a253942f22f1748ccae506622275c,d97d84561ac65b1a
Normal-1988.png,Normal,Normal/images/Normal-1988.png,Normal/masks/Normal-1988.png,True,G10705,train,acef147e125e6ad0f1ef6193cd468e65,9969061a9f7ab0ea
Normal-1989.png,Normal,Normal/images/Normal-1989.png,Normal/masks/Normal-1989.png,True,G10706,train,727564e30ad691e2f0c14f800abde6be,dc3d534b43ca2ac3
Normal-199.png,Normal,Normal/images/Normal-199.png,Normal/masks/Normal-199.png,True,G10707,train,ae47cc835fa6c470eeb384d289122868,d12cc71b8eca39d4
Normal-1990.png,Normal,Normal/images/Normal-1990.png,Normal/masks/Normal-1990.png,True,G10708,train,709492dd76432b750915626c8b5026ec,813f061e1f4f1c5e
Normal-1991.png,Normal,Normal/images/Normal-1991.png,Normal/masks/Normal-1991.png,True,G10709,train,f33ab7c18d1d4e1952b88fbb1226588c,d43d43471f4a2c3e
Normal-1992.png,Normal,Normal/images/Normal-1992.png,Normal/masks/Normal-1992.png,True,G10710,train,ed71e229a849bcb3c093df374415e23d,cc7cc34b10431fbd
Normal-1993.png,Normal,Normal/images/Normal-1993.png,Normal/masks/Normal-1993.png,True,G10711,train,62c1ca0799b4dabf1be175fe546d3cf1,8178061e1d5d7d5d
Normal-1994.png,Normal,Normal/images/Normal-1994.png,Normal/masks/Normal-1994.png,True,G10712,train,9c4e20bf70af8a1b64efd2acd6086eb1,c93f57471e0a491e
Normal-1995.png,Normal,Normal/images/Normal-1995.png,Normal/masks/Normal-1995.png,True,G10713,train,abd0e63fde889ea458b8704a4efd24f9,987c871b094d3e5d
Normal-1996.png,Normal,Normal/images/Normal-1996.png,Normal/masks/Normal-1996.png,True,G10714,train,729644ba33de1694080ec2ff392637c9,d1acc7194d1c784f
Normal-1997.png,Normal,Normal/images/Normal-1997.png,Normal/masks/Normal-1997.png,True,G10715,val,de28c52c7d1af4170ea7eada8d60e5df,ddacc31b840f3d0e
Normal-1998.png,Normal,Normal/images/Normal-1998.png,Normal/masks/Normal-1998.png,True,G10716,train,5d3e633d642cb12151dd0bf9057170d9,d93dc64213474e1f
Normal-1999.png,Normal,Normal/images/Normal-1999.png,Normal/masks/Normal-1999.png,True,G10717,val,a5b9da8eb4c85dc061b2ae8267b24fb9,d96f061a49e5f8c4
Normal-2.png,Normal,Normal/images/Normal-2.png,Normal/masks/Normal-2.png,True,G10718,train,8235b119f2f546a6621e93b39e889b16,913b57161a1e4d7a
Normal-20.png,Normal,Normal/images/Normal-20.png,Normal/masks/Normal-20.png,True,G10719,train,40a1dee058c4eddd9d450e342c3e412d,d03b421b1f6d3e38
Normal-200.png,Normal,Normal/images/Normal-200.png,Normal/masks/Normal-200.png,True,G10720,train,f5e2c05f2b1d983deb05eab033e9ed23,d0bec71bc8c93ac2
Normal-2000.png,Normal,Normal/images/Normal-2000.png,Normal/masks/Normal-2000.png,True,G10721,val,ab218982ac5c2f6181b13131846c30be,d91f461e1d481f5c
Normal-2001.png,Normal,Normal/images/Normal-2001.png,Normal/masks/Normal-2001.png,True,G10722,test,0a702eaee3416f4e2c1456501a8580df,dfb1805e317a5ea0
Normal-2002.png,Normal,Normal/images/Normal-2002.png,Normal/masks/Normal-2002.png,True,G10723,test,7ebc9da4be03a18a8307b1eba3bd06f1,8c0d431b6d7e7272
Normal-2003.png,Normal,Normal/images/Normal-2003.png,Normal/masks/Normal-2003.png,True,G10724,train,3514992363f681661c6323e63642c0c6,81797e867a9165a5
Normal-2004.png,Normal,Normal/images/Normal-2004.png,Normal/masks/Normal-2004.png,True,G10725,test,5f673a43e8ff586fb07b729d7607f528,897805161e1f5f5b
Normal-2005.png,Normal,Normal/images/Normal-2005.png,Normal/masks/Normal-2005.png,True,G10726,train,faa0f6c9dcf229bff2b5acdd2c96cbbf,d97b85161e0d5a78
Normal-2006.png,Normal,Normal/images/Normal-2006.png,Normal/masks/Normal-2006.png,True,G10727,train,9c16fdc852f2aa4f7067ff4473de0772,9969c71e944e3a9a
Normal-2007.png,Normal,Normal/images/Normal-2007.png,Normal/masks/Normal-2007.png,True,G10728,train,f9b3250ae6a394fa4279458dbba110f3,cc34d34b0b0bcf2d
Normal-2008.png,Normal,Normal/images/Normal-2008.png,Normal/masks/Normal-2008.png,True,G10729,train,bafd119f144fe31d1f09817c67bd89f2,dc2fc25d422b992d
Normal-2009.png,Normal,Normal/images/Normal-2009.png,Normal/masks/Normal-2009.png,True,G10730,train,a8cf23d58d7ef3635a9d8bea13bb9f0e,dd3d424f090c2fbc
Normal-201.png,Normal,Normal/images/Normal-201.png,Normal/masks/Normal-201.png,True,G10731,val,60cd547ab5b124c3a13f7873498deee3,94fc973bad0d282c
Normal-2010.png,Normal,Normal/images/Normal-2010.png,Normal/masks/Normal-2010.png,True,G10732,test,9a086f4a9400863848da16743491b42f,dd2ad20b0f842cbf
Normal-2011.png,Normal,Normal/images/Normal-2011.png,Normal/masks/Normal-2011.png,True,G10733,test,95dc6aa7b16a1f84c1330dee8315b9a5,996196161b9ece5a
Normal-2012.png,Normal,Normal/images/Normal-2012.png,Normal/masks/Normal-2012.png,True,G10734,train,66d08eba3d1cd6e43821e0e88f237806,897d861a1d4d38ed
Normal-2013.png,Normal,Normal/images/Normal-2013.png,Normal/masks/Normal-2013.png,True,G10735,train,ff105328f1bdb4795d1a5c5bfc60c64d,996dc71a0a4f3ab8
Normal-2014.png,Normal,Normal/images/Normal-2014.png,Normal/masks/Normal-2014.png,True,G10736,train,68737b2568a0416a4225a704d3132c91,d92ec41a17ce1abc
Normal-2015.png,Normal,Normal/images/Normal-2015.png,Normal/masks/Normal-2015.png,True,G10737,train,47b4dd012418b3f4585b4955d81c5158,dd2cd4c20b5d8f43
Normal-2016.png,Normal,Normal/images/Normal-2016.png,Normal/masks/Normal-2016.png,True,G10738,train,f1591ad17ab6d0682916252625e85d89,817a964a191d6b3f
Normal-2017.png,Normal,Normal/images/Normal-2017.png,Normal/masks/Normal-2017.png,True,G10739,train,c14432f2dcf4f1efcae7fe553b7de9c8,d92f8656094f3a69
Normal-2018.png,Normal,Normal/images/Normal-2018.png,Normal/masks/Normal-2018.png,True,G10740,train,e54f75c51ab889e5d5453831d7e4f455,987d851617471f1d
Normal-2019.png,Normal,Normal/images/Normal-2019.png,Normal/masks/Normal-2019.png,True,G10741,val,cbf78c8e9ed70c4dc08281bd63cfac47,897d865e1e475b18
Normal-202.png,Normal,Normal/images/Normal-202.png,Normal/masks/Normal-202.png,True,G10742,val,2b9952adc686cb240ed515e5738a2242,902aca5adc5d5cd6
Normal-2020.png,Normal,Normal/images/Normal-2020.png,Normal/masks/Normal-2020.png,True,G10743,test,0ea8029175011e1649561228baff939f,dd68961a4c2ab8ed
Normal-2021.png,Normal,Normal/images/Normal-2021.png,Normal/masks/Normal-2021.png,True,G10744,val,a83e7cce1705a47d9eaf692570d6b367,f915061e451b5d5b
Normal-2022.png,Normal,Normal/images/Normal-2022.png,Normal/masks/Normal-2022.png,True,G10745,train,909cf8b80cc4a574faa0b72a5e37dd02,807f85561e4d5b1b
Normal-2023.png,Normal,Normal/images/Normal-2023.png,Normal/masks/Normal-2023.png,True,G10746,train,f70378bdfed8eba706746043db2925f2,996d471a494f194f
Normal-2024.png,Normal,Normal/images/Normal-2024.png,Normal/masks/Normal-2024.png,True,G10747,train,a3797d9895b1bbde0bee0ae2d63ab729,9c2dd552074d1e5d
Normal-2025.png,Normal,Normal/images/Normal-2025.png,Normal/masks/Normal-2025.png,True,G10748,train,8464ec15330fa495dd5a11e5dec7e6f1,d92ad61a5a5b194b
Normal-2026.png,Normal,Normal/images/Normal-2026.png,Normal/masks/Normal-2026.png,True,G10749,train,7fcda4f7ae9f654496323d2aefac56bc,9979071e1ecd4e38
Normal-2027.png,Normal,Normal/images/Normal-2027.png,Normal/masks/Normal-2027.png,True,G10750,train,c4211a69e9a6ff85f864439e0f27128e,d979d69a45d259c0
Normal-2028.png,Normal,Normal/images/Normal-2028.png,Normal/masks/Normal-2028.png,True,G10751,train,a40977f3b43ab4332d5e68bbe213ad83,d43f531b0b382b2e
Normal-2029.png,Normal,Normal/images/Normal-2029.png,Normal/masks/Normal-2029.png,True,G10752,train,110e01bae505fde3df937aa48e3bfc0c,9c6c421b194d3ebd
Normal-203.png,Normal,Normal/images/Normal-203.png,Normal/masks/Normal-203.png,True,G10753,train,b0cb1f8f9152baaa2f8bb04eb08aa2ad,955f871b850476e6
Normal-2030.png,Normal,Normal/images/Normal-2030.png,Normal/masks/Normal-2030.png,True,G10754,train,cdcfd302a410d28e811305e702ec5185,887f87061a5f5a5a
Normal-2031.png,Normal,Normal/images/Normal-2031.png,Normal/masks/Normal-2031.png,True,G10755,train,8962f96a2094b6faa7a32225bf01d11b,d03d4b4b2c8a2f4f
Normal-2032.png,Normal,Normal/images/Normal-2032.png,Normal/masks/Normal-2032.png,True,G10756,train,31fe443292673d20101093af0df30b63,9468075b4f2e4b6d
Normal-2033.png,Normal,Normal/images/Normal-2033.png,Normal/masks/Normal-2033.png,True,G10757,train,c4e24d1152c655d80da5ca7b507e5457,bc6cc2031b4d4f1f
Normal-2034.png,Normal,Normal/images/Normal-2034.png,Normal/masks/Normal-2034.png,True,G10758,train,e2194e03d73e5ce4eda8a7aad5ff8baa,d42d535b4a4d0a5f
Normal-2035.png,Normal,Normal/images/Normal-2035.png,Normal/masks/Normal-2035.png,True,G10759,val,e81be60fc7e753e862c11e4e13f9b317,ec2ec30b0f1a0f6e
Normal-2036.png,Normal,Normal/images/Normal-2036.png,Normal/masks/Normal-2036.png,True,G10760,test,59536aca6004fe841a8a5a095640bb29,997e471a050f996d
Normal-2037.png,Normal,Normal/images/Normal-2037.png,Normal/masks/Normal-2037.png,True,G10761,test,55fb113f9151dadf11055cd79bcf66d9,d138961b5a5b195d
Normal-2038.png,Normal,Normal/images/Normal-2038.png,Normal/masks/Normal-2038.png,True,G10762,train,234c5a53123cc4f4f4a9a803178881ef,dc29d25b4a435f58
Normal-2039.png,Normal,Normal/images/Normal-2039.png,Normal/masks/Normal-2039.png,True,G10763,train,43f49a35d28aeb50e0743e7b0f501cbc,d42e534b0a4f1f2e
Normal-204.png,Normal,Normal/images/Normal-204.png,Normal/masks/Normal-204.png,True,G10764,val,f67cb9c37c58d90ee59b2ce288687b8f,902bc33ad85b1dcb
Normal-2040.png,Normal,Normal/images/Normal-2040.png,Normal/masks/Normal-2040.png,True,G10765,train,2e13b0fc1be47286f4749955bb9d882a,917b471819adb8d5
Normal-2041.png,Normal,Normal/images/Normal-2041.png,Normal/masks/Normal-2041.png,True,G10766,test,53790d5b1d30732fadcce1463d07fadd,81fac659d6ce1e50
Normal-2042.png,Normal,Normal/images/Normal-2042.png,Normal/masks/Normal-2042.png,True,G10767,train,fcfe44126422e06e96c7617d3b6cf707,d13dc6071b944e3b
Normal-2043.png,Normal,Normal/images/Normal-2043.png,Normal/masks/Normal-2043.png,True,G10768,train,298da9dbaf1b1178c193a42d2dc7acf2,9979c71a194e3a2b
Normal-2044.png,Normal,Normal/images/Normal-2044.png,Normal/masks/Normal-2044.png,True,G10769,train,07d8361aa4e993383123f894af5abcd9,897ac75e194d2d38
Normal-2045.png,Normal,Normal/images/Normal-2045.png,Normal/masks/Normal-2045.png,True,G10770,train,c8d8344bccff25121d23a70ef26113d8,d83d52171e483f39
Normal-2046.png,Normal,Normal/images/Normal-2046.png,Normal/masks/Normal-2046.png,True,G10771,train,99372006d22984f5883eaa78043accfb,dc2ec30f4e0e2c6e
Normal-2047.png,Normal,Normal/images/Normal-2047.png,Normal/masks/Normal-2047.png,True,G10772,val,0352e4c1bc095fff0be81840797be417,c95f421f482b38af
Normal-2048.png,Normal,Normal/images/Normal-2048.png,Normal/masks/Normal-2048.png,True,G10773,train,b96423888cf0c70d0f41efab0be893dc,997b841a1f485b3d
Normal-2049.png,Normal,Normal/images/Normal-2049.png,Normal/masks/Normal-2049.png,True,G10774,train,bf2de8ee002d34fbe49f545dbdcf6c56,997d8616145f1a5d
Normal-205.png,Normal,Normal/images/Normal-205.png,Normal/masks/Normal-205.png,True,G10775,train,8a502efa59803017d4c3941b5b7c5aae,912bc61adc791cce
Normal-2050.png,Normal,Normal/images/Normal-2050.png,Normal/masks/Normal-2050.png,True,G10776,train,b97ef60a2545ba5970dbae46cb3f36b8,d97d461b194e193c
Normal-2051.png,Normal,Normal/images/Normal-2051.png,Normal/masks/Normal-2051.png,True,G10777,train,2a69cbdcd1882d98ff9ae810d3d63ca5,ce2d52534d4b4b0f
Normal-2052.png,Normal,Normal/images/Normal-2052.png,Normal/masks/Normal-2052.png,True,G10778,train,886e18b9276ae87dd78bbc1867dd5934,997f861a851e194f
Normal-2053.png,Normal,Normal/images/Normal-2053.png,Normal/masks/Normal-2053.png,True,G10779,train,43408cf4be892211494a2ce01b431b9e,87781b5f595959c0
Normal-2054.png,Normal,Normal/images/Normal-2054.png,Normal/masks/Normal-2054.png,True,G10780,train,296644a9e82b545c615f2fee93413fd6,d53fc70e19451b38
Normal-2055.png,Normal,Normal/images/Normal-2055.png,Normal/masks/Normal-2055.png,True,G10781,val,5dc37ce8d61026e295e025b32a414ff0,9939c61a1d465f1d
Normal-2056.png,Normal,Normal/images/Normal-2056.png,Normal/masks/Normal-2056.png,True,G10782,val,f03944bea85f1c6353c5a2d7cd717230,9d2d061a596e78f8
Normal-2057.png,Normal,Normal/images/Normal-2057.png,Normal/masks/Normal-2057.png,True,G10783,test,232fde0ea69379a34e3534e88d428e22,817b161a5f5e3989
Normal-2058.png,Normal,Normal/images/Normal-2058.png,Normal/masks/Normal-2058.png,True,G10784,test,caf2f10f4e54a212ceb2e271994ddf16,c9398617194d5f5c
Normal-2059.png,Normal,Normal/images/Normal-2059.png,Normal/masks/Normal-2059.png,True,G10785,test,7c7172072a5abbbf2161371fcb3e0806,987f841e591f184f
Normal-206.png,Normal,Normal/images/Normal-206.png,Normal/masks/Normal-206.png,True,G10786,train,a9a5ed93abd16509e4affa0063d23147,d47aa73dad8c6884
Normal-2060.png,Normal,Normal/images/Normal-2060.png,Normal/masks/Normal-2060.png,True,G10787,train,6edebdca0a864cde4ec99e562ee9c9b5,897d47021d4d0f7b
Normal-2061.png,Normal,Normal/images/Normal-2061.png,Normal/masks/Normal-2061.png,True,G10788,train,e477e7364709d118a6fb4162080b5399,81694e1e5a5afa69
Normal-2062.png,Normal,Normal/images/Normal-2062.png,Normal/masks/Normal-2062.png,True,G10789,train,4e310e7bc3db547af673d08602e6fb54,8079461f1b6d39b9
Normal-2063.png,Normal,Normal/images/Normal-2063.png,Normal/masks/Normal-2063.png,True,G10790,train,1eb2cf1d1bb2d56470f2cd36498b33b4,896806191f4fb8fb
Normal-2064.png,Normal,Normal/images/Normal-2064.png,Normal/masks/Normal-2064.png,True,G10791,train,47d3778acafa7193d31bc9b1ed5895c5,817d1e1a5de8e0e5
Normal-2065.png,Normal,Normal/images/Normal-2065.png,Normal/masks/Normal-2065.png,True,G10792,train,ff86937bd757b1ba66f395960c23a809,c87c46191b2f3cb5
Normal-2066.png,Normal,Normal/images/Normal-2066.png,Normal/masks/Normal-2066.png,True,G10793,test,01117ae0e7212763e70cc4c3a74531df,8168961a195f7b79
Normal-2067.png,Normal,Normal/images/Normal-2067.png,Normal/masks/Normal-2067.png,True,G10794,train,e7b2549c3fabfed5a76621df6c8ac404,f945161a1f39293d
Normal-2068.png,Normal,Normal/images/Normal-2068.png,Normal/masks/Normal-2068.png,True,G10795,train,d9dec0e1eb6aa662b5d739c17053abc8,817847161f5d7a39
Normal-2069.png,Normal,Normal/images/Normal-2069.png,Normal/masks/Normal-2069.png,True,G10796,train,d66ecf11dca13ac5962a41cd09a37542,dd3d424b1dac28bc
Normal-207.png,Normal,Normal/images/Normal-207.png,Normal/masks/Normal-207.png,True,G10797,train,5ff9ff7ab15b962d3676d42be8a4c45f,d06ec70dcd6529c3
Normal-2070.png,Normal,Normal/images/Normal-2070.png,Normal/masks/Normal-2070.png,True,G10798,train,01e113b29d26126dbb49e677bccf13cf,997856181939b99f
Normal-2071.png,Normal,Normal/images/Normal-2071.png,Normal/masks/Normal-2071.png,True,G10799,val,31b885aac5a14babb6377c4428beb9f7,cd2c42c30bc50fdf
Normal-2072.png,Normal,Normal/images/Normal-2072.png,Normal/masks/Normal-2072.png,True,G10800,val,0a38e087a2cd6cf0eb4f76bd21c0f727,d93c861b114f1ade
Normal-2073.png,Normal,Normal/images/Normal-2073.png,Normal/masks/Normal-2073.png,True,G10801,train,64ebfb84c7e2e5331407b82a328f0f09,9968061b595e5d5b
Normal-2074.png,Normal,Normal/images/Normal-2074.png,Normal/masks/Normal-2074.png,True,G10802,train,b0a2a62ebd1382e728bc97616bba7839,881f061e1c5b3beb
Normal-2075.png,Normal,Normal/images/Normal-2075.png,Normal/masks/Normal-2075.png,True,G10803,val,7164450e0d9534547f4d779597b41ad2,d03d46471b1d6c79
Normal-2076.png,Normal,Normal/images/Normal-2076.png,Normal/masks/Normal-2076.png,True,G10804,train,2582a1a0b4bf16fb392fcf851f6eed2e,d4b5830f4bc30f9a
Normal-2077.png,Normal,Normal/images/Normal-2077.png,Normal/masks/Normal-2077.png,True,G10805,val,bf54a8845f601922a22e15ea85b292d6,9979c7061e934b1d
Normal-2078.png,Normal,Normal/images/Normal-2078.png,Normal/masks/Normal-2078.png,True,G10806,test,88745830dc8b9964f3e806e8e435694c,896f061a4f5a195f
Normal-2079.png,Normal,Normal/images/Normal-2079.png,Normal/masks/Normal-2079.png,True,G10807,train,a4fad02dfcf66524d2915fa8c554dd9c,9c6cc25b4bc51b59
Normal-208.png,Normal,Normal/images/Normal-208.png,Normal/masks/Normal-208.png,True,G10808,train,72dfe21f57da6861344d2ad6f1b9c058,946e831f0b0f1d5d
Normal-2080.png,Normal,Normal/images/Normal-2080.png,Normal/masks/Normal-2080.png,True,G10809,test,7fc20d7a488a0b1a73e5f45494a8ef58,cc2dc242bf0aaf0f
Normal-2081.png,Normal,Normal/images/Normal-2081.png,Normal/masks/Normal-2081.png,True,G10810,train,fdaf4d140614a6f97eab6bc4e4240423,883f960b1f451f1d
Normal-2082.png,Normal,Normal/images/Normal-2082.png,Normal/masks/Normal-2082.png,True,G10811,train,b0808f45028ef1540dabf2aa04997bcd,913fc6065a5d4b1e
Normal-2083.png,Normal,Normal/images/Normal-2083.png,Normal/masks/Normal-2083.png,True,G10812,train,456d81d5dec530ecccd65bfe4ea8982c,99311e59da69b1c5
Normal-2084.png,Normal,Normal/images/Normal-2084.png,Normal/masks/Normal-2084.png,True,G10813,train,6e053ad9228444ada7a406c2571eff23,dc2ec20b1d463dbc
Normal-2085.png,Normal,Normal/images/Normal-2085.png,Normal/masks/Normal-2085.png,True,G10814,train,8b6467c952336c5e7b71d08ae61dfe67,d93b461a944bb9e8
Normal-2086.png,Normal,Normal/images/Normal-2086.png,Normal/masks/Normal-2086.png,True,G10815,train,c9aa924460093d77f253cf3b215278e3,d56c02870b4efd1d
Normal-2087.png,Normal,Normal/images/Normal-2087.png,Normal/masks/Normal-2087.png,True,G10816,train,22421dca9f3826a963c97398651df175,dc3ec24d164a3e6c
Normal-2088.png,Normal,Normal/images/Normal-2088.png,Normal/masks/Normal-2088.png,True,G10817,train,eaac6c8a67df43ba5a91ddde232c4f42,897a07125b4d787b
Normal-2089.png,Normal,Normal/images/Normal-2089.png,Normal/masks/Normal-2089.png,True,G10818,train,231d7fd1521c9948891b7da4482e9ba5,9979071e58555a79
Normal-209.png,Normal,Normal/images/Normal-209.png,Normal/masks/Normal-209.png,True,G10819,train,5a011c91e913629e93bba7f9a79ef631,907bc71ac93d3919
Normal-2090.png,Normal,Normal/images/Normal-2090.png,Normal/masks/Normal-2090.png,True,G10820,val,4770866d9b88b5dcbbe2199ee3c5e780,bd4cc31f4d1c4d46
Normal-2091.png,Normal,Normal/images/Normal-2091.png,Normal/masks/Normal-2091.png,True,G10821,val,82dbc362cedb77cfc2a862eeb507306e,dd2cc20bdd4f1a1c
Normal-2092.png,Normal,Normal/images/Normal-2092.png,Normal/masks/Normal-2092.png,True,G10822,test,39245344be2989164843270313e2f6d1,807dc60f1ad64f1d
Normal-2093.png,Normal,Normal/images/Normal-2093.png,Normal/masks/Normal-2093.png,True,G10823,test,ab3388f0429679230bd60fb65a10e585,9969870616d75e17
Normal-2094.png,Normal,Normal/images/Normal-2094.png,Normal/masks/Normal-2094.png,True,G10824,test,d019104c359767ff8a4abceef4362108,817f061e594d6e78
Normal-2095.png,Normal,Normal/images/Normal-2095.png,Normal/masks/Normal-2095.png,True,G10825,train,40c501f80ff2a092d3cea124af93f9af,95695a5a99ab858d
Normal-2096.png,Normal,Normal/images/Normal-2096.png,Normal/masks/Normal-2096.png,True,G10826,train,3827df8c5b0c3d374dffeba94a0b9e1a,897f061e594d3969
Normal-2097.png,Normal,Normal/images/Normal-2097.png,Normal/masks/Normal-2097.png,True,G10827,train,c86add939e800bc4866fc11419b21bc9,d969059e12cf5d58
Normal-2098.png,Normal,Normal/images/Normal-2098.png,Normal/masks/Normal-2098.png,True,G10828,val,57b702727d3a77ab8df98ec24bc32080,9d2cc01b154f1edd
Normal-2099.png,Normal,Normal/images/Normal-2099.png,Normal/masks/Normal-2099.png,True,G10829,train,6f1c760ba9b17a8d8658f0a614717ced,957887124bc71e5b
Normal-21.png,Normal,Normal/images/Normal-21.png,Normal/masks/Normal-21.png,True,G10830,train,274e19e38e69e0e03737a8339d338eee,d92b461a0e3db4e5
Normal-210.png,Normal,Normal/images/Normal-210.png,Normal/masks/Normal-210.png,True,G10831,train,f8c57a10f314c266d24f8f57de7a4f6d,d13ec71a5d483dd0
Normal-2100.png,Normal,Normal/images/Normal-2100.png,Normal/masks/Normal-2100.png,True,G10832,train,e686078549ac844e0d41b6879a664e26,cc3cc34b5dc22f2c
Normal-2101.png,Normal,Normal/images/Normal-2101.png,Normal/masks/Normal-2101.png,True,G10833,train,fd4a3507b3a561945a31b24757dfc55a,c978121f1ece3ab8
Normal-2102.png,Normal,Normal/images/Normal-2102.png,Normal/masks/Normal-2102.png,True,G10834,train,dfa758133eb29ce5eb392292fda24ec8,dc2d42430f0e6f3e
Normal-2103.png,Normal,Normal/images/Normal-2103.png,Normal/masks/Normal-2103.png,True,G10835,train,b3795aed2ef1a41e84bcc2370d947904,917b57104b4f1b0f
Normal-2104.png,Normal,Normal/images/Normal-2104.png,Normal/masks/Normal-2104.png,True,G10836,train,406d11a7388e36a0eda066e333b161de,f96882131e461bfb
Normal-2105.png,Normal,Normal/images/Normal-2105.png,Normal/masks/Normal-2105.png,True,G10837,train,420e747a4e9a38f16fb7e26f0df4733a,9847861e475e587e
Normal-2106.png,Normal,Normal/images/Normal-2106.png,Normal/masks/Normal-2106.png,True,G10838,train,3120267276a46f4dd90abc3880589c4a,d47c464b190e6e3d
Normal-2107.png,Normal,Normal/images/Normal-2107.png,Normal/masks/Normal-2107.png,True,G10839,train,e357f2fb182381020af90553d30be84d,d929e01b1dc0cf9e
Normal-2108.png,Normal,Normal/images/Normal-2108.png,Normal/masks/Normal-2108.png,True,G10840,val,6092037a37f82062095c7895b404ddfe,dd28530b5d481f5d
Normal-2109.png,Normal,Normal/images/Normal-2109.png,Normal/masks/Normal-2109.png,True,G10841,val,467bad81a8c45edab588ac7304003279,913b16161e1f5f1a
Normal-211.png,Normal,Normal/images/Normal-211.png,Normal/masks/Normal-211.png,True,G10842,test,52fb92ef2f152e4b011fbe2fd4dcfac6,d43ee33f9c183c90
Normal-2110.png,Normal,Normal/images/Normal-2110.png,Normal/masks/Normal-2110.png,True,G10843,test,b379086c005e31da1549fca704801365,d96c86390f4e494f
Normal-2111.png,Normal,Normal/images/Normal-2111.png,Normal/masks/Normal-2111.png,True,G10844,train,c03166ef0da9d99d062314a3868e65d4,99681e1a5d792d69
Normal-2112.png,Normal,Normal/images/Normal-2112.png,Normal/masks/Normal-2112.png,True,G10845,test,01c2cb42b87ac7f3dfdab22fd897d7cf,dd3dc24b0b0e2b3c
Normal-2113.png,Normal,Normal/images/Normal-2113.png,Normal/masks/Normal-2113.png,True,G10846,train,113d0051e4e15c29416caba6d80414cb,d93d42021f1bd64f
Normal-2114.png,Normal,Normal/images/Normal-2114.png,Normal/masks/Normal-2114.png,True,G10847,train,7fdaedb0546cfd57d98b7c174d01b60a,d42dc3d35d460f0e
Normal-2115.png,Normal,Normal/images/Normal-2115.png,Normal/masks/Normal-2115.png,True,G10848,train,95167955ae0550aa3d89e4fd4721f35d,c57b861a1a4f5b38
Normal-2116.png,Normal,Normal/images/Normal-2116.png,Normal/masks/Normal-2116.png,True,G10849,train,4327057cd96adaa8dec3018cb9f9817f,dc3dc21d140f3ccb
Normal-2117.png,Normal,Normal/images/Normal-2117.png,Normal/masks/Normal-2117.png,True,G10850,train,ecad641ccdfe766ebdd8d4c5b7025e96,d90c464a4f2a3f6e
Normal-2118.png,Normal,Normal/images/Normal-2118.png,Normal/masks/Normal-2118.png,True,G10851,train,8621eb7ca00cb9ebb91bc60020ed515c,b921c65a585d595d
Normal-2119.png,Normal,Normal/images/Normal-2119.png,Normal/masks/Normal-2119.png,True,G10852,train,c72a91ae30f8dbef25a05ac812f4a257,dd2ac21d48585d6f
Normal-212.png,Normal,Normal/images/Normal-212.png,Normal/masks/Normal-212.png,True,G10853,train,7983b6819aae89d416f0298a5977024e,912bc61ad8693bd6
Normal-2120.png,Normal,Normal/images/Normal-2120.png,Normal/masks/Normal-2120.png,True,G10854,train,38ccbbd8b50f81c2bfc826487a076b30,8978061a5f487bf9
Normal-2121.png,Normal,Normal/images/Normal-2121.png,Normal/masks/Normal-2121.png,True,G10855,train,994d7fad54a75cd9b4086483de048891,9768d62d65963983
Normal-2122.png,Normal,Normal/images/Normal-2122.png,Normal/masks/Normal-2122.png,True,G10856,val,ca0ea6f6c1aad56bfff9605ec56e5fa0,807d5f57480a0f6f
Normal-2123.png,Normal,Normal/images/Normal-2123.png,Normal/masks/Normal-2123.png,True,G10857,train,d2e653b6cab788862b444af1d65bdda9,8178defa5a4d281d
Normal-2124.png,Normal,Normal/images/Normal-2124.png,Normal/masks/Normal-2124.png,True,G10858,train,057253e8d199fbf38bf72a81274f67fa,993e465b154f1a4d
Normal-2125.png,Normal,Normal/images/Normal-2125.png,Normal/masks/Normal-2125.png,True,G10859,train,9ee94dab7aec4121a8eb5cbdbb54df73,c93d87461a487dba
Normal-2126.png,Normal,Normal/images/Normal-2126.png,Normal/masks/Normal-2126.png,True,G10860,val,d7d04b41a80ec08aa377952cb5f9af78,817a47535f0e7b88
Normal-2127.png,Normal,Normal/images/Normal-2127.png,Normal/masks/Normal-2127.png,True,G10861,val,6fa2338cad06b84820d6250b531f8204,d02d46d61f0edc1b
Normal-2128.png,Normal,Normal/images/Normal-2128.png,Normal/masks/Normal-2128.png,True,G10862,train,fc6bee817032abce62f96c8e82b967bd,993ec61a5d195e1a
Normal-2129.png,Normal,Normal/images/Normal-2129.png,Normal/masks/Normal-2129.png,True,G10863,train,885062a76c7c251637375191588b5850,8939161e5dee58b8
Normal-213.png,Normal,Normal/images/Normal-213.png,Normal/masks/Normal-213.png,True,G10864,test,64ea058c9f79033ddd3015ba5040241f,907da71aad0d7cc4
Normal-2130.png,Normal,Normal/images/Normal-2130.png,Normal/masks/Normal-2130.png,True,G10865,test,c8155e78b6b8a5c92c053e81c5508949,dd2e425b196d283d
Normal-2131.png,Normal,Normal/images/Normal-2131.png,Normal/masks/Normal-2131.png,True,G10866,test,0d31236a2570f83354ff8d28a8d13c5e,993f061a1b7d6a58
Normal-2132.png,Normal,Normal/images/Normal-2132.png,Normal/masks/Normal-2132.png,True,G10867,train,9c399c8845a7af7580a647340f350236,997d851e18c579b1
Normal-2133.png,Normal,Normal/images/Normal-2133.png,Normal/masks/Normal-2133.png,True,G10868,train,a2a3c8415042c787383fd0f589725ff0,8178464e195f6d3b
Normal-2134.png,Normal,Normal/images/Normal-2134.png,Normal/masks/Normal-2134.png,True,G10869,train,44b6d2b7c1281d1b6e9ab46e2495b789,887f421f0a2d1eaf
Normal-2135.png,Normal,Normal/images/Normal-2135.png,Normal/masks/Normal-2135.png,True,G10870,val,667c03640de05e8c6066e6ba15abfd0b,d87905161e595b1f
Normal-2136.png,Normal,Normal/images/Normal-2136.png,Normal/masks/Normal-2136.png,True,G10871,train,d2ed9a5ed962a305415f326c2c4cfd4c,d97bd61e1c481a39
Normal-2137.png,Normal,Normal/images/Normal-2137.png,Normal/masks/Normal-2137.png,True,G10872,train,d7ca4b9790351bec5b4807a034c0b09c,dd2fc2571b40af18
Normal-2138.png,Normal,Normal/images/Normal-2138.png,Normal/masks/Normal-2138.png,True,G10873,train,dbe716ed737acadb7662b3582c2c2a04,997b861e184d3e1e
Normal-2139.png,Normal,Normal/images/Normal-2139.png,Normal/masks/Normal-2139.png,True,G10874,train,6cc582f778c3493174ed42fbae96a7bc,c83d470b1e4a6f3c
Normal-214.png,Normal,Normal/images/Normal-214.png,Normal/masks/Normal-214.png,True,G10875,val,d8ba251f54ffccefc882363cb678ae15,91f4ee1ae99d2909
Normal-2140.png,Normal,Normal/images/Normal-2140.png,Normal/masks/Normal-2140.png,True,G10876,test,6f4f6fd384f261081e589a0346d27cc2,ed6c50d2434f4b0f
Normal-2141.png,Normal,Normal/images/Normal-2141.png,Normal/masks/Normal-2141.png,True,G10877,train,978b4b569bb1304d23e0ae30214c7157,d93d464b186d6d98
Normal-2142.png,Normal,Normal/images/Normal-2142.png,Normal/masks/Normal-2142.png,True,G10878,train,775d50fe2186ed1851cbc8f7cd49cd41,852e03d31f09c7de
Normal-2143.png,Normal,Normal/images/Normal-2143.png,Normal/masks/Normal-2143.png,True,G10879,test,2e00cf24931831409a1fe6934fd55bb3,d92d461b0c4ebde1
Normal-2144.png,Normal,Normal/images/Normal-2144.png,Normal/masks/Normal-2144.png,True,G10880,train,e9a14982793cb13645035107f5ac93ad,987b051e1e5f5a19
Normal-2145.png,Normal,Normal/images/Normal-2145.png,Normal/masks/Normal-2145.png,True,G10881,train,cdf394986bca0cb2c16eaa6dbbabda01,dc2f42570b0e3db8
Normal-2146.png,Normal,Normal/images/Normal-2146.png,Normal/masks/Normal-2146.png,True,G10882,train,a0ee341bfb55ea4266506e17d8688c87,d97b061b195919ce
Normal-2147.png,Normal,Normal/images/Normal-2147.png,Normal/masks/Normal-2147.png,True,G10883,val,5bd2b6bf175a53e3c11e251d774eb464,996cc71b0b4e78c5
Normal-2148.png,Normal,Normal/images/Normal-2148.png,Normal/masks/Normal-2148.png,True,G10884,test,fba4afd877b0b0d293f722288bfb78a4,9979c696184dde1a
Normal-2149.png,Normal,Normal/images/Normal-2149.png,Normal/masks/Normal-2149.png,True,G10885,train,f22cb2b0b10efc64af0b5068d0a7b880,dc2bc10f0a2b3eb3
Normal-215.png,Normal,Normal/images/Normal-215.png,Normal/masks/Normal-215.png,True,G10886,train,79eecea08d10a3d2203be6efb6a53169,d064c73dcc3c3cc6
Normal-2150.png,Normal,Normal/images/Normal-2150.png,Normal/masks/Normal-2150.png,True,G10887,train,c3cf0bfbdcc7e811c48d21d2ce73dfc1,8178179a3b39acb5
Normal-2151.png,Normal,Normal/images/Normal-2151.png,Normal/masks/Normal-2151.png,True,G10888,train,793936dee80415c2cf76b9425b715da4,d97dc61a114f1b1c
Normal-2152.png,Normal,Normal/images/Normal-2152.png,Normal/masks/Normal-2152.png,True,G10889,val,1fa38342ac76d82097a48790452dc0f4,993fc6161a454f1e
Normal-2153.png,Normal,Normal/images/Normal-2153.png,Normal/masks/Normal-2153.png,True,G10890,train,b7cf984b4a25565cb3f328d353b2d467,cc3f425d1a6c1e6c
Normal-2154.png,Normal,Normal/images/Normal-2154.png,Normal/masks/Normal-2154.png,True,G10891,train,23aa28fd3d37fac5923e7fc8a00d1fc9,dd3d501f0f461e19
Normal-2155.png,Normal,Normal/images/Normal-2155.png,Normal/masks/Normal-2155.png,True,G10892,train,d4385ab680b77548af0b28b6301b60ae,dd2cc2531b494f1e
Normal-2156.png,Normal,Normal/images/Normal-2156.png,Normal/masks/Normal-2156.png,True,G10893,train,3d3621d2eadbad5e42362a4e92feace6,cd45834f4d1f0f2a
Normal-2157.png,Normal,Normal/images/Normal-2157.png,Normal/masks/Normal-2157.png,True,G10894,train,d0e0e95158f92fa6d0456e5e3774a5ba,d83d461b1b0c7e39
Normal-2158.png,Normal,Normal/images/Normal-2158.png,Normal/masks/Normal-2158.png,True,G10895,train,b2756a68039157512d023d95e9781caf,dd2cc31b4d0a3de8
Normal-2159.png,Normal,Normal/images/Normal-2159.png,Normal/masks/Normal-2159.png,True,G10896,val,2630b1d3d9f1ae902ec407b4af5f4f12,dc2cc25bd1cb4bd0
Normal-216.png,Normal,Normal/images/Normal-216.png,Normal/masks/Normal-216.png,True,G10897,train,bbcd911dac3782402f600a049cafd7ba,d05ee719e9c51ac2
Normal-2160.png,Normal,Normal/images/Normal-2160.png,Normal/masks/Normal-2160.png,True,G10898,train,3d4a1c44f408f1b40634b50b6e49d692,d97b46065ec03b9c
Normal-2161.png,Normal,Normal/images/Normal-2161.png,Normal/masks/Normal-2161.png,True,G10899,train,420204a07902a8b9fcdb25a79b69372d,dc2d42d70b644f78
Normal-2162.png,Normal,Normal/images/Normal-2162.png,Normal/masks/Normal-2162.png,True,G10900,val,480ed3d68a400affab23880b9a2581f9,d96846191f6939e5
Normal-2163.png,Normal,Normal/images/Normal-2163.png,Normal/masks/Normal-2163.png,True,G10901,test,b8d0508cea1aba4b7ceb147804e4cbac,957916865a5f4a59
Normal-2164.png,Normal,Normal/images/Normal-2164.png,Normal/masks/Normal-2164.png,True,G10902,train,7410f266a65a35fc8d240392c88781fe,d52c42df3ec42bd0
Normal-2165.png,Normal,Normal/images/Normal-2165.png,Normal/masks/Normal-2165.png,True,G10903,test,5ffde266daa917084af340c89db8837a,817f06161a1b7f39
Normal-2166.png,Normal,Normal/images/Normal-2166.png,Normal/masks/Normal-2166.png,True,G10904,test,bd445f7062136753ab9aaf0140840ebe,d13946c64b19ae4f
Normal-2167.png,Normal,Normal/images/Normal-2167.png,Normal/masks/Normal-2167.png,True,G10905,test,8af10c2aef084ec7751b23e4b94e08d3,996ed619441b7947
Normal-2168.png,Normal,Normal/images/Normal-2168.png,Normal/masks/Normal-2168.png,True,G10906,test,86e1acba2cbe07a02b821767ce986704,c52e434b0d4c1f5f
Normal-2169.png,Normal,Normal/images/Normal-2169.png,Normal/masks/Normal-2169.png,True,G10907,train,6a9c4df2bec9242444d9e0435d9aa3d6,d57d46c21a4a475b
Normal-217.png,Normal,Normal/images/Normal-217.png,Normal/masks/Normal-217.png,True,G10908,train,9ae2be880df4a2fc403cba18df66027d,812cc31bdc6d39cd
Normal-2170.png,Normal,Normal/images/Normal-2170.png,Normal/masks/Normal-2170.png,True,G10909,train,7a4bda94027fba519a3dd803bfb25f09,d939061a79e978f0
Normal-2171.png,Normal,Normal/images/Normal-2171.png,Normal/masks/Normal-2171.png,True,G10910,train,a6ca9d1a650d81ed47a9c9ac6fa79eb0,d97dc71e18491b19
Normal-2172.png,Normal,Normal/images/Normal-2172.png,Normal/masks/Normal-2172.png,True,G10911,train,b1a876a3d5d31dd63b63d1ef645aae50,992dc4129b471f1f
Normal-2173.png,Normal,Normal/images/Normal-2173.png,Normal/masks/Normal-2173.png,True,G10912,train,400b6cecf26143c8cb75b55bf8e0fc7a,d43f424f5d4e0f0a
Normal-2174.png,Normal,Normal/images/Normal-2174.png,Normal/masks/Normal-2174.png,True,G10913,train,7ea3b0f7efbaa1472e015ff1ea4e9f81,996816584d79cde6
Normal-2175.png,Normal,Normal/images/Normal-2175.png,Normal/masks/Normal-2175.png,True,G10914,train,6d64df42a4003f9643807030698b0f49,916916d71a1e5f1a
Normal-2176.png,Normal,Normal/images/Normal-2176.png,Normal/masks/Normal-2176.png,True,G10915,train,e5c795064d2a60206ccdbfe9cdb2d983,817cc6581d4e395f
Normal-2177.png,Normal,Normal/images/Normal-2177.png,Normal/masks/Normal-2177.png,True,G10916,train,c237c0afe00db3db9943f17f04f08cd8,9d6cc61a4b071eb9
Normal-2178.png,Normal,Normal/images/Normal-2178.png,Normal/masks/Normal-2178.png,True,G10917,train,8c509bc4490609fb8e6c71ad302a46d6,dd2ed05393454f16
Normal-2179.png,Normal,Normal/images/Normal-2179.png,Normal/masks/Normal-2179.png,True,G10918,train,b68ef36048aaa56b09a2ddf290aa13a0,996d061e471e175e
Normal-218.png,Normal,Normal/images/Normal-218.png,Normal/masks/Normal-218.png,True,G10919,train,41c26866f464e99221bc9ec9c1ab281c,917f8e1a190f1ac7
Normal-2180.png,Normal,Normal/images/Normal-2180.png,Normal/masks/Normal-2180.png,True,G10920,train,c91742d252d41f836a05067e08dde236,dd29d2571d4a0b39
Normal-2181.png,Normal,Normal/images/Normal-2181.png,Normal/masks/Normal-2181.png,True,G10921,val,81c440c90b2f61457db73c6383c744d4,9c3cd05397531575
Normal-2182.png,Normal,Normal/images/Normal-2182.png,Normal/masks/Normal-2182.png,True,G10922,val,eb1c50c00719f64fa4f5520ce09b75d3,f96dc58a1e055e1c
Normal-2183.png,Normal,Normal/images/Normal-2183.png,Normal/masks/Normal-2183.png,True,G10923,val,62f79daafc06c1b4346f56271fe154d5,d93dc20f15c64f1c
Normal-2184.png,Normal,Normal/images/Normal-2184.png,Normal/masks/Normal-2184.png,True,G10924,test,e06bf2316292aba5a85ce931e98af075,d928d7d71e0c4a1d
Normal-2185.png,Normal,Normal/images/Normal-2185.png,Normal/masks/Normal-2185.png,True,G10925,train,1f813d5f9987683084bdc509214d5f38,9c3d460b1f4d4e1e
Normal-2186.png,Normal,Normal/images/Normal-2186.png,Normal/masks/Normal-2186.png,True,G10926,train,c5a2e45c9c1599ea08e96313fa95f746,99685f190dbc9197
Normal-2187.png,Normal,Normal/images/Normal-2187.png,Normal/masks/Normal-2187.png,True,G10927,train,f5b9052cfca55f82f718425247a0e009,9d3e164b1b4c6d38
Normal-2188.png,Normal,Normal/images/Normal-2188.png,Normal/masks/Normal-2188.png,True,G10928,train,b68c8feede7a5a4a70935bde4c6a4787,8178165a5acf7939
Normal-2189.png,Normal,Normal/images/Normal-2189.png,Normal/masks/Normal-2189.png,True,G10929,train,da559e7af22bd9d9755a09624a447b91,9979071acd1ab8e9
Normal-219.png,Normal,Normal/images/Normal-219.png,Normal/masks/Normal-219.png,True,G10930,train,92358c55f4733ee9381053a61efe1e45,956e8f2f0d4c6cc4
Normal-2190.png,Normal,Normal/images/Normal-2190.png,Normal/masks/Normal-2190.png,True,G10931,train,79aabd586c068cbddeb7b5b22bae3159,d93dc21d950e3a0f
Normal-2191.png,Normal,Normal/images/Normal-2191.png,Normal/masks/Normal-2191.png,True,G10932,train,1dff1c6e19d34e2aae18b54d57ca8ea1,817906065b1fee4f
Normal-2192.png,Normal,Normal/images/Normal-2192.png,Normal/masks/Normal-2192.png,True,G10933,train,0b49477dbc9ea7b4144627a981b46753,d80d045e586b7be9
Normal-2193.png,Normal,Normal/images/Normal-2193.png,Normal/masks/Normal-2193.png,True,G10934,train,9ab0ccd9a02d1c1909c46b3cb9d38f32,ed3cd24b1a470f1a
Normal-2194.png,Normal,Normal/images/Normal-2194.png,Normal/masks/Normal-2194.png,True,G10935,train,e18b33544d57ea61f4efb3e511f4c474,dd2ad21b51483b5d
Normal-2195.png,Normal,Normal/images/Normal-2195.png,Normal/masks/Normal-2195.png,True,G10936,train,b1198959aaa08878a19a01f356d31a12,ccbdc34fc3cb0b80
Normal-2196.png,Normal,Normal/images/Normal-2196.png,Normal/masks/Normal-2196.png,True,G10937,train,82f415725ab7df31653db16eff776e8d,d43d4a435f4d4d0e
Normal-2197.png,Normal,Normal/images/Normal-2197.png,Normal/masks/Normal-2197.png,True,G10938,val,8114950756858f0286d8d303ce2bbcdb,887b07d61e8c5f1a
Normal-2198.png,Normal,Normal/images/Normal-2198.png,Normal/masks/Normal-2198.png,True,G10939,val,f5b3a2d2f5d0c36773671d91480ba024,887f861e1d1f194d
Normal-2199.png,Normal,Normal/images/Normal-2199.png,Normal/masks/Normal-2199.png,True,G10940,train,a4600121969ada1d7d17415f6c9dcb9e,d9390646193d693f
Normal-22.png,Normal,Normal/images/Normal-22.png,Normal/masks/Normal-22.png,True,G10941,test,4b84e9e1bdfcee711c87ab0e5f81743f,81f8f69416877ad1
Normal-220.png,Normal,Normal/images/Normal-220.png,Normal/masks/Normal-220.png,True,G10942,val,ff0232761a2bf32251f9fa69dc984a29,d46e871b892d3c2d
Normal-2200.png,Normal,Normal/images/Normal-2200.png,Normal/masks/Normal-2200.png,True,G10943,test,00e9aada01aabbbbd6577a3e8cba8ffa,897f86187de980e5
Normal-2201.png,Normal,Normal/images/Normal-2201.png,Normal/masks/Normal-2201.png,True,G10944,train,17d6e071465199652278a68a694ebeda,d92d84571f4e4a1d
Normal-2202.png,Normal,Normal/images/Normal-2202.png,Normal/masks/Normal-2202.png,True,G10945,test,4b576517a36d629b559dd6a8f63a2c64,c03d4ad7591b452f
Normal-2203.png,Normal,Normal/images/Normal-2203.png,Normal/masks/Normal-2203.png,True,G10946,test,713df57c7aae677e17c376a6470327ec,9968c651171e795d
Normal-2204.png,Normal,Normal/images/Normal-2204.png,Normal/masks/Normal-2204.png,True,G10947,train,fadaf11d7e41a8ebec08be5ff66eb1c0,8138971f5a5e195b
Normal-2205.png,Normal,Normal/images/Normal-2205.png,Normal/masks/Normal-2205.png,True,G10948,train,59e964fd8c0de8cb43ff9f1cf3390eda,d42dc34f0e2f4a1e
Normal-2206.png,Normal,Normal/images/Normal-2206.png,Normal/masks/Normal-2206.png,True,G10949,train,fc798b2634da9c0ff9f4a28fa4efec7d,d17f87d25a495858
Normal-2207.png,Normal,Normal/images/Normal-2207.png,Normal/masks/Normal-2207.png,True,G10950,train,78e83a0c22df074a8addcbac2f7b7a5c,d93dc60e184e6f38
Normal-2208.png,Normal,Normal/images/Normal-2208.png,Normal/masks/Normal-2208.png,True,G10951,val,b4aab8a321dafa9499ed02c387384db7,987b47861a0f3e3a
Normal-2209.png,Normal,Normal/images/Normal-2209.png,Normal/masks/Normal-2209.png,True,G10952,val,16f772d7da0f13dbdf6b3803b1acb50b,dd2dc20b1d4c1b3d
Normal-221.png,Normal,Normal/images/Normal-221.png,Normal/masks/Normal-221.png,True,G10953,train,fe839b214d7a51076decff68259087b6,d47e871f0d852ccc
Normal-2210.png,Normal,Normal/images/Normal-2210.png,Normal/masks/Normal-2210.png,True,G10954,train,aea522b5a6d3f18d17a95cb742f74145,c02c1bd32fce299d
Normal-2211.png,Normal,Normal/images/Normal-2211.png,Normal/masks/Normal-2211.png,True,G10955,train,cf7bc078e19ba99428c0b1a481c73e4c,dc2b431f4b0d1c2f
Normal-2212.png,Normal,Normal/images/Normal-2212.png,Normal/masks/Normal-2212.png,True,G10956,test,9fa16b7f6faba2e458783ecd4183f315,9879061e1b4f5b1b
Normal-2213.png,Normal,Normal/images/Normal-2213.png,Normal/masks/Normal-2213.png,True,G10957,train,bd0db3defea336e15660a18099a629bc,9161861e1f175e7a
Normal-2214.png,Normal,Normal/images/Normal-2214.png,Normal/masks/Normal-2214.png,True,G10958,train,c3704afbf84ecf05786baac89c73ad0c,8179055e5ae9787a
Normal-2215.png,Normal,Normal/images/Normal-2215.png,Normal/masks/Normal-2215.png,True,G10959,train,b4b81e9e07b78711a0f1be95bd019bda,957c075a1b6c78f0
Normal-2216.png,Normal,Normal/images/Normal-2216.png,Normal/masks/Normal-2216.png,True,G10960,train,6ede57915e04a897e43e67641e88fa56,dd2cc24f4d4e0c7c
Normal-2217.png,Normal,Normal/images/Normal-2217.png,Normal/masks/Normal-2217.png,True,G10961,train,22d6afb193a712e59fcd1acf5114eee2,dd2a564b5d444b1d
Normal-2218.png,Normal,Normal/images/Normal-2218.png,Normal/masks/Normal-2218.png,True,G10962,train,8d437a801d98927bf23ad0939fd4d234,d928071f57075d4d
Normal-2219.png,Normal,Normal/images/Normal-2219.png,Normal/masks/Normal-2219.png,True,G10963,train,7e88af332ca87432532558b4e935fe42,cc0ed70b4d0c4dcf
Normal-222.png,Normal,Normal/images/Normal-222.png,Normal/masks/Normal-222.png,True,G10964,val,03cd90aa6f606d309ccc894bff8ee7e6,d43eef1f14141ec4
Normal-2220.png,Normal,Normal/images/Normal-2220.png,Normal/masks/Normal-2220.png,True,G10965,test,36050ff74bf56d6279db68f432722db0,9138165b79a16bb9
Normal-2221.png,Normal,Normal/images/Normal-2221.png,Normal/masks/Normal-2221.png,True,G10966,test,1d2f2a497de0b554b916b0b0ae0c9ed4,9161a616f916d739
Normal-2222.png,Normal,Normal/images/Normal-2222.png,Normal/masks/Normal-2222.png,True,G10967,train,9f3eca4dcbb31c0608ef55809597844b,897f061e151a5f0f
Normal-2223.png,Normal,Normal/images/Normal-2223.png,Normal/masks/Normal-2223.png,True,G10968,train,30eccbd25eb66deb9ed273d082998a86,ea3c474b1a4b4f2c
Normal-2224.png,Normal,Normal/images/Normal-2224.png,Normal/masks/Normal-2224.png,True,G10969,train,49a0803c5d885fa5f1653148467e7700,cc3ed34b5f481d44
Normal-2225.png,Normal,Normal/images/Normal-2225.png,Normal/masks/Normal-2225.png,True,G10970,train,1a930cd6c03199024719494fedc4b77e,cc74d34f5c4a2e4a
Normal-2226.png,Normal,Normal/images/Normal-2226.png,Normal/masks/Normal-2226.png,True,G10971,train,6a88b4410e34760e98ab2fac8fc9dc45,d97dc71e954e1c40
Normal-2227.png,Normal,Normal/images/Normal-2227.png,Normal/masks/Normal-2227.png,True,G10972,train,39d13e0c1b9ad35cd298d3f665c99b5e,d42d43570f0e2d3d
Normal-2228.png,Normal,Normal/images/Normal-2228.png,Normal/masks/Normal-2228.png,True,G10973,train,2a6bbabf096997043bb321bbb5012d8a,9978c7161e451f1e
Normal-2229.png,Normal,Normal/images/Normal-2229.png,Normal/masks/Normal-2229.png,True,G10974,train,4bf62e398b7c7d4c6adc9ea5466cfa94,9969061a1f5a39eb
Normal-223.png,Normal,Normal/images/Normal-223.png,Normal/masks/Normal-223.png,True,G10975,train,71e699b347303e7a307f03e6da372c85,957c8f1d8d0d4cc6
Normal-2230.png,Normal,Normal/images/Normal-2230.png,Normal/masks/Normal-2230.png,True,G10976,train,e4770cf7f923163ff152895afd8ac79f,817dc60a0f5f195e
Normal-2231.png,Normal,Normal/images/Normal-2231.png,Normal/masks/Normal-2231.png,True,G10977,train,d3874df456d07d3f77564b25cf42bf18,f949d61a594938e9
Normal-2232.png,Normal,Normal/images/Normal-2232.png,Normal/masks/Normal-2232.png,True,G10978,test,a8437a1a627598166b94c0e6e79b0f56,912906181f5f79fa
Normal-2233.png,Normal,Normal/images/Normal-2233.png,Normal/masks/Normal-2233.png,True,G10979,train,434949bc2da176c9a8ebdd919f9a5b5b,d85d430b083d3dbd
Normal-2234.png,Normal,Normal/images/Normal-2234.png,Normal/masks/Normal-2234.png,True,G10980,train,6b712689bd130588662556ea3e856903,8979571e1a4d1b39
Normal-2235.png,Normal,Normal/images/Normal-2235.png,Normal/masks/Normal-2235.png,True,G10981,val,497d0148d228f478ebe4b443da2e78da,817b16125f2a5e2f
Normal-2236.png,Normal,Normal/images/Normal-2236.png,Normal/masks/Normal-2236.png,True,G10982,val,563fa297674ee6f0905a721a7751aa24,9978c61b156db8ac
Normal-2237.png,Normal,Normal/images/Normal-2237.png,Normal/masks/Normal-2237.png,True,G10983,val,ce4cd2d46f6e5e32a536703e38d32a04,d93d061bc44a39ed
Normal-2238.png,Normal,Normal/images/Normal-2238.png,Normal/masks/Normal-2238.png,True,G10984,train,1ca9ac6717efddd3c4ad4567f49a8462,dd2850835b1d5e5b
Normal-2239.png,Normal,Normal/images/Normal-2239.png,Normal/masks/Normal-2239.png,True,G10985,train,8ecd15cf3789828ca27683b3922994b8,dc2ec2431f0a4f5e
Normal-224.png,Normal,Normal/images/Normal-224.png,Normal/masks/Normal-224.png,True,G10986,train,1cbf8a80b5f3e7f9257059ae0fb8deb1,957e9f5885168a87
Normal-2240.png,Normal,Normal/images/Normal-2240.png,Normal/masks/Normal-2240.png,True,G10987,test,ae1fbb84209b50afceac0a641070df09,8c7d825b49ed6878
Normal-2241.png,Normal,Normal/images/Normal-2241.png,Normal/masks/Normal-2241.png,True,G10988,test,c2e9878d22b453c66611a18ffa51e4c3,dd2cc21b944b9acb
Normal-2242.png,Normal,Normal/images/Normal-2242.png,Normal/masks/Normal-2242.png,True,G10989,test,006af6c6485aaa96d3e21b7401e940e6,cc3ec34b1c4e0f3c
Normal-2243.png,Normal,Normal/images/Normal-2243.png,Normal/masks/Normal-2243.png,True,G10990,train,b71d271da84905f216bee59933bd1506,897bd6121a497b39
Normal-2244.png,Normal,Normal/images/Normal-2244.png,Normal/masks/Normal-2244.png,True,G10991,train,e62e8179410502a1275537b9877499c3,d96d461f451e154b
Normal-2245.png,Normal,Normal/images/Normal-2245.png,Normal/masks/Normal-2245.png,True,G10992,train,de8b95678cc21710345484aaffbefb43,897d061a5d5c787c
Normal-2246.png,Normal,Normal/images/Normal-2246.png,Normal/masks/Normal-2246.png,True,G10993,train,978592126baffaaddd40a23417fc8cd5,994c861a4e4a7cef
Normal-2247.png,Normal,Normal/images/Normal-2247.png,Normal/masks/Normal-2247.png,True,G10994,train,88294d97529de54b47c0a87ac8fffdf6,85917a9e9561a5e5
Normal-2248.png,Normal,Normal/images/Normal-2248.png,Normal/masks/Normal-2248.png,True,G10995,train,c75364d61e7e2eefda4beaebca3a5526,c02bf6c6435d5c69
Normal-2249.png,Normal,Normal/images/Normal-2249.png,Normal/masks/Normal-2249.png,True,G10996,val,d60867cd405b0a7e64ccc1edae79f477,dd2dd24a1f4e0a1d
Normal-225.png,Normal,Normal/images/Normal-225.png,Normal/masks/Normal-225.png,True,G10997,train,de4332838a1f6f8241f10eb212cb053a,c020c34bbeea3ff0
Normal-2250.png,Normal,Normal/images/Normal-2250.png,Normal/masks/Normal-2250.png,True,G10998,train,a14206fd944346cc4c8315133a197a4e,dc3d42534f4a0f1b
Normal-2251.png,Normal,Normal/images/Normal-2251.png,Normal/masks/Normal-2251.png,True,G10999,train,8240e788ef787d5f0d951446fb4dfa8a,dc28431f3cf43ab4
Normal-2252.png,Normal,Normal/images/Normal-2252.png,Normal/masks/Normal-2252.png,True,G11000,train,30a71a161859d1f4ade6310a987f4b7d,8178961a18dfb9b9
Normal-2253.png,Normal,Normal/images/Normal-2253.png,Normal/masks/Normal-2253.png,True,G11001,train,423351a8b38b9eeb684a18beeb5102ac,887d8716594d5f58
Normal-2254.png,Normal,Normal/images/Normal-2254.png,Normal/masks/Normal-2254.png,True,G11002,val,ac145a8342220dfc346fe9715ff21e00,d928c61a5d4d39f8
Normal-2255.png,Normal,Normal/images/Normal-2255.png,Normal/masks/Normal-2255.png,True,G11003,test,34b5e2c658744f9ec1d505c5b0f8f5f4,d879865e98475a59
Normal-2256.png,Normal,Normal/images/Normal-2256.png,Normal/masks/Normal-2256.png,True,G11004,val,d5727b509608f58c019e27d7b003f9cd,817b04de5979a1e5
Normal-2257.png,Normal,Normal/images/Normal-2257.png,Normal/masks/Normal-2257.png,True,G11005,test,360909e7de66eb016a45d79147a9500f,dd2c860b1f8e4b1d
Normal-2258.png,Normal,Normal/images/Normal-2258.png,Normal/masks/Normal-2258.png,True,G11006,train,0ca1e8ac811f642fe97d79b45cb243d0,d96f065e1a4c5ae8
Normal-2259.png,Normal,Normal/images/Normal-2259.png,Normal/masks/Normal-2259.png,True,G11007,train,d15391336e58002f1456938ba2df3b3b,d93d864619174e79
Normal-226.png,Normal,Normal/images/Normal-226.png,Normal/masks/Normal-226.png,True,G11008,train,bac4c497c7466f8c9f7e122b39c3b1fb,916fc74bcd4938c2
Normal-2260.png,Normal,Normal/images/Normal-2260.png,Normal/masks/Normal-2260.png,True,G11009,train,2165d1783d23b9368607587472a00bff,997d071a4d4e7869
Normal-2261.png,Normal,Normal/images/Normal-2261.png,Normal/masks/Normal-2261.png,True,G11010,train,d081abaaa984d067c29232517f797d83,9969c71a09453ef6
Normal-2262.png,Normal,Normal/images/Normal-2262.png,Normal/masks/Normal-2262.png,True,G11011,train,e00c41f0938bbb75f5323df84ff241c5,9968061a5d4d79f9
Normal-2263.png,Normal,Normal/images/Normal-2263.png,Normal/masks/Normal-2263.png,True,G11012,train,21b61fa900c0a635ae22f9eb5643c1d4,996c861a974f3a8d
Normal-2264.png,Normal,Normal/images/Normal-2264.png,Normal/masks/Normal-2264.png,True,G11013,train,570cfeede787edf2da5f58f464641557,cc3dc2d31bc04b3d
Normal-2265.png,Normal,Normal/images/Normal-2265.png,Normal/masks/Normal-2265.png,True,G11014,val,c88a47981e6c4112fda33d692d100dd9,d97f425b1b4d0a38
Normal-2266.png,Normal,Normal/images/Normal-2266.png,Normal/masks/Normal-2266.png,True,G11015,val,44a77b0a0ecd3131a8b7d9256038fcaf,fc2ad21f4a525e4a
Normal-2267.png,Normal,Normal/images/Normal-2267.png,Normal/masks/Normal-2267.png,True,G11016,train,368ceec9a86770de82e2fde0f47e9b4c,d42f414b0f4e4e4f
Normal-2268.png,Normal,Normal/images/Normal-2268.png,Normal/masks/Normal-2268.png,True,G11017,train,ef33c05f6ee860c9c78e1a260ed974c2,996d461a1ae94e7a
Normal-2269.png,Normal,Normal/images/Normal-2269.png,Normal/masks/Normal-2269.png,True,G11018,train,89c057f76f23fdcca3c2bdb584379413,d969c69ed4463a51
Normal-227.png,Normal,Normal/images/Normal-227.png,Normal/masks/Normal-227.png,True,G11019,train,f88fbe46a2ec1e66238d3c7222bac1a9,c07cc61e9dcd29e4
Normal-2270.png,Normal,Normal/images/Normal-2270.png,Normal/masks/Normal-2270.png,True,G11020,train,4b9ab7344aeb094b0a810b7b80997c90,916847961e195d6f
Normal-2271.png,Normal,Normal/images/Normal-2271.png,Normal/masks/Normal-2271.png,True,G11021,train,5e4b22f54827bb0621a27ff1fc40bc60,dc2f071b1d4c5c5c
Normal-2272.png,Normal,Normal/images/Normal-2272.png,Normal/masks/Normal-2272.png,True,G11022,train,fae8a0a7dce675454155b27c4c4b5eb6,fd7cd34b4e001b8a
Normal-2273.png,Normal,Normal/images/Normal-2273.png,Normal/masks/Normal-2273.png,True,G11023,test,b24d518902d44b4bbcbea05a717b17a7,99691e181b5e7879
Normal-2274.png,Normal,Normal/images/Normal-2274.png,Normal/masks/Normal-2274.png,True,G11024,val,ee405078e052ac2ae98d2cfb138efa4c,dd2cd24b1b424b1f
Normal-2275.png,Normal,Normal/images/Normal-2275.png,Normal/masks/Normal-2275.png,True,G11025,test,ba92bbc9ff5d1c8e9b11ba1b51ae858d,dc3c425b8d0a0fbd
Normal-2276.png,Normal,Normal/images/Normal-2276.png,Normal/masks/Normal-2276.png,True,G11026,test,40b6dd2127c298e2c3a5ec3cb0f49be1,dc2ad30f4c4e1d4b
Normal-2277.png,Normal,Normal/images/Normal-2277.png,Normal/masks/Normal-2277.png,True,G11027,train,2ae36eb606073f2cbcc8a4f69dc7a730,c13d87421d1e4e6f
Normal-2278.png,Normal,Normal/images/Normal-2278.png,Normal/masks/Normal-2278.png,True,G11028,train,deb6325edfc42d7510fe4733a7cf9ac1,817807841adf78df
Normal-2279.png,Normal,Normal/images/Normal-2279.png,Normal/masks/Normal-2279.png,True,G11029,train,785907acac996b2a71b85e1695846db4,8179d6c6da90c8cf
Normal-228.png,Normal,Normal/images/Normal-228.png,Normal/masks/Normal-228.png,True,G11030,train,d67dfdc409f65ad82a58d8b22641620a,913de71ab94838dc
Normal-2280.png,Normal,Normal/images/Normal-2280.png,Normal/masks/Normal-2280.png,True,G11031,train,f8e45f4330550e88cc30ae78095e0a12,c83dc35f5c403db2
Normal-2281.png,Normal,Normal/images/Normal-2281.png,Normal/masks/Normal-2281.png,True,G11032,train,16199def1ebf7a3408457ff5d9530a5b,817c0e1a6abd78b5
Normal-2282.png,Normal,Normal/images/Normal-2282.png,Normal/masks/Normal-2282.png,True,G11033,train,3e44e8d01d24c61b3e940b280faf12ed,d92d861a5b4e5a39
Normal-2283.png,Normal,Normal/images/Normal-2283.png,Normal/masks/Normal-2283.png,True,G11034,train,9c93c1e788f63c0b72f50fb62d3479a9,897b871e155e18c7
Normal-2284.png,Normal,Normal/images/Normal-2284.png,Normal/masks/Normal-2284.png,True,G11035,train,d475ac69bf1eb6c1fa684112c0251e91,d53dc25f526b5a40
Normal-2285.png,Normal,Normal/images/Normal-2285.png,Normal/masks/Normal-2285.png,True,G11036,train,ebc8592eb79e0bcf39a0cd60e467dbb6,c939c61f1c6a38f8
Normal-2286.png,Normal,Normal/images/Normal-2286.png,Normal/masks/Normal-2286.png,True,G11037,train,1952fcafe1220f27c2238d17ad2e9b50,9969871687571c57
Normal-2287.png,Normal,Normal/images/Normal-2287.png,Normal/masks/Normal-2287.png,True,G11038,val,ae8287be749516dfe8d6468a5978ab96,b93dc6ca194d4e58
Normal-2288.png,Normal,Normal/images/Normal-2288.png,Normal/masks/Normal-2288.png,True,G11039,test,817c4300833dc30129193e6b9a711c8f,882f061e455f78ed
Normal-2289.png,Normal,Normal/images/Normal-2289.png,Normal/masks/Normal-2289.png,True,G11040,train,3fe99e852441fe7792968aee6e5934f3,c03d430b3c2fb4bd
Normal-229.png,Normal,Normal/images/Normal-229.png,Normal/masks/Normal-229.png,True,G11041,train,94ae5f5202409245204fa02999f30957,d07ec719cc0db8c6
Normal-2290.png,Normal,Normal/images/Normal-2290.png,Normal/masks/Normal-2290.png,True,G11042,test,ec7e9b4c9d34d1612976e54082be9216,ddfcd21bd0433843
Normal-2291.png,Normal,Normal/images/Normal-2291.png,Normal/masks/Normal-2291.png,True,G11043,train,6d74004cffbcbb225df24ec00bbce3dd,dd2cc21bdb421ed8
Normal-2292.png,Normal,Normal/images/Normal-2292.png,Normal/masks/Normal-2292.png,True,G11044,val,abca291367c8fc3691b68be4ac5c0ce2,817bc61e19497f19
Normal-2293.png,Normal,Normal/images/Normal-2293.png,Normal/masks/Normal-2293.png,True,G11045,val,20d44a9de5f17cfcc55fdd993e56980f,cc3dd04a0bcb8b8f
Normal-2294.png,Normal,Normal/images/Normal-2294.png,Normal/masks/Normal-2294.png,True,G11046,test,119b0b52da8fc99dd7dd9a78400cc975,dd2cd20a5f550b1d
Normal-2295.png,Normal,Normal/images/Normal-2295.png,Normal/masks/Normal-2295.png,True,G11047,train,29473a3580dd925623b7b7e1dcb6ad83,dc3c425b184e3ebc
Normal-2296.png,Normal,Normal/images/Normal-2296.png,Normal/masks/Normal-2296.png,True,G11048,train,9da9a48b891a8f9d106e370c0d1018d2,dc2f035d4c4a5b4b
Normal-2297.png,Normal,Normal/images/Normal-2297.png,Normal/masks/Normal-2297.png,True,G11049,train,d4a9f80ad481def3614e9d91c2b1608e,9969c65e12c55e33
Normal-2298.png,Normal,Normal/images/Normal-2298.png,Normal/masks/Normal-2298.png,True,G11050,train,43186581a4d0d2565aaebcffbe9bbf93,d42f43570b0e2d2f
Normal-2299.png,Normal,Normal/images/Normal-2299.png,Normal/masks/Normal-2299.png,True,G11051,train,3c7d4716ba886955fd8fd96cbc4fbbec,d95d47561e4d5a18
Normal-23.png,Normal,Normal/images/Normal-23.png,Normal/masks/Normal-23.png,True,G11052,train,91bf65f8e44ea4d04d0da8e3ca4a142d,c93f85c59615471e
Normal-230.png,Normal,Normal/images/Normal-230.png,Normal/masks/Normal-230.png,True,G11053,train,53087209378c519ab03ff7eb72a2cf5f,9178867ab8ce39e1
Normal-2300.png,Normal,Normal/images/Normal-2300.png,Normal/masks/Normal-2300.png,True,G11054,train,2100ad2ba362752f2b6d507b06ef6b08,d92bc7071f445e58
Normal-2301.png,Normal,Normal/images/Normal-2301.png,Normal/masks/Normal-2301.png,True,G11055,train,e4bccecb4069adc346e4ae8e2582035f,996886181d6f79e9
Normal-2302.png,Normal,Normal/images/Normal-2302.png,Normal/masks/Normal-2302.png,True,G11056,train,b53d44dc8c19214280d3e0012fc03736,993d061e5c6b79e0
Normal-2303.png,Normal,Normal/images/Normal-2303.png,Normal/masks/Normal-2303.png,True,G11057,train,1e202dee2a24c7d4027fc6729de961d0,99780616171f1f1f
Normal-2304.png,Normal,Normal/images/Normal-2304.png,Normal/masks/Normal-2304.png,True,G11058,train,ef249446bbfb2bc2eed35524e274ef8a,d635420b4e2f2e6e
Normal-2305.png,Normal,Normal/images/Normal-2305.png,Normal/masks/Normal-2305.png,True,G11059,train,118401463660f70ff8853a3cd19f7d6e,8c4f474b0b4d2e7c
Normal-2306.png,Normal,Normal/images/Normal-2306.png,Normal/masks/Normal-2306.png,True,G11060,val,1cae6f265f131c835707b13d436085e2,d93d031f1aca3c39
Normal-2307.png,Normal,Normal/images/Normal-2307.png,Normal/masks/Normal-2307.png,True,G11061,val,bf7d1799fb036e6128bef5a8ae84d390,cc1e8b4dae283cf4
Normal-2308.png,Normal,Normal/images/Normal-2308.png,Normal/masks/Normal-2308.png,True,G11062,val,9b7a61e400db0b5e014f1f832c0a46fd,c9ba4792074f1e4e
Normal-2309.png,Normal,Normal/images/Normal-2309.png,Normal/masks/Normal-2309.png,True,G11063,test,17dc2719931781be63324008edc94d3f,dd2ad31fc21f1641
Normal-231.png,Normal,Normal/images/Normal-231.png,Normal/masks/Normal-231.png,True,G11064,test,5455313cae150a9f6947d04571e0eb71,91f6ce1ac92d61c6
Normal-2310.png,Normal,Normal/images/Normal-2310.png,Normal/masks/Normal-2310.png,True,G11065,test,c2fb818c5327082e498ca2d7fbaa7dad,dc2e425b146a7d71
Normal-2311.png,Normal,Normal/images/Normal-2311.png,Normal/masks/Normal-2311.png,True,G11066,train,e2b5ae05217dcd98ac576bc2e0a08a79,d97f565e1a1a1c0a
Normal-2312.png,Normal,Normal/images/Normal-2312.png,Normal/masks/Normal-2312.png,True,G11067,train,a1bd153a87a633789a610a59e6caf09a,d43c43cb0f2ce53c
Normal-2313.png,Normal,Normal/images/Normal-2313.png,Normal/masks/Normal-2313.png,True,G11068,train,a5168761f511cdc843fe79253004da80,c9340f0f59daec38
Normal-2314.png,Normal,Normal/images/Normal-2314.png,Normal/masks/Normal-2314.png,True,G11069,train,890dc6bc102b22c49ae4d1cd2175a1d1,9968c75b455f184d
Normal-2315.png,Normal,Normal/images/Normal-2315.png,Normal/masks/Normal-2315.png,True,G11070,train,353e28b061de24ef4e5d02b79eccf0ff,cc3dc2438b0f42bf
Normal-2316.png,Normal,Normal/images/Normal-2316.png,Normal/masks/Normal-2316.png,True,G11071,train,41a3cc9997374ea1e6a55f2baabe5687,936014121f5f5b5f
Normal-2317.png,Normal,Normal/images/Normal-2317.png,Normal/masks/Normal-2317.png,True,G11072,train,4f428e327a975b9c7913177afd494d2f,c62ef1d10df0d85c
Normal-2318.png,Normal,Normal/images/Normal-2318.png,Normal/masks/Normal-2318.png,True,G11073,train,d9f3d72b7a0f0b0093afd255fa2ca3af,d43fc3530b494f1a
Normal-2319.png,Normal,Normal/images/Normal-2319.png,Normal/masks/Normal-2319.png,True,G11074,val,dfa262c9253688aec0d2a27f82fa6979,8168861a5b4f5f5b
Normal-232.png,Normal,Normal/images/Normal-232.png,Normal/masks/Normal-232.png,True,G11075,train,19f0685775d59537df192dabedd3edea,81beee9a594f1341
Normal-2320.png,Normal,Normal/images/Normal-2320.png,Normal/masks/Normal-2320.png,True,G11076,train,59d6c205c4cbdb07733464b09039f874,993f465a195d5d0c
Normal-2321.png,Normal,Normal/images/Normal-2321.png,Normal/masks/Normal-2321.png,True,G11077,test,94f1570c6f185dcbe04bb40b656632c6,987dc7561b0d7f00
Normal-2322.png,Normal,Normal/images/Normal-2322.png,Normal/masks/Normal-2322.png,True,G11078,test,a3ca396a56e74c4b3f51404b0080871d,9d0e1719a5b8c3cb
Normal-2323.png,Normal,Normal/images/Normal-2323.png,Normal/masks/Normal-2323.png,True,G11079,train,a68171b24be0f040a12613fcac1be268,897987161d475f19
Normal-2324.png,Normal,Normal/images/Normal-2324.png,Normal/masks/Normal-2324.png,True,G11080,train,2607510e5b5c19b68250293d092fb0af,d979071e1a4f381b
Normal-2325.png,Normal,Normal/images/Normal-2325.png,Normal/masks/Normal-2325.png,True,G11081,train,e9cadef163592a67b6e207840bb21568,8d3d861a1f4d1ea9
Normal-2326.png,Normal,Normal/images/Normal-2326.png,Normal/masks/Normal-2326.png,True,G11082,train,6a53a056bbaa009ba1c8af4ba5313cb1,9d3e464f1f401f58
Normal-2327.png,Normal,Normal/images/Normal-2327.png,Normal/masks/Normal-2327.png,True,G11083,val,1c6969ebf11a203b1dc7d2e283a201ef,d16b021d5f4a393d
Normal-2328.png,Normal,Normal/images/Normal-2328.png,Normal/masks/Normal-2328.png,True,G11084,val,65e1816be3b51a956c8c7f95669dbb3b,d93906561f1d4f19
Normal-2329.png,Normal,Normal/images/Normal-2329.png,Normal/masks/Normal-2329.png,True,G11085,train,cb1bd444bcba15d000dd9733581047b7,91691e1f79e86961
Normal-233.png,Normal,Normal/images/Normal-233.png,Normal/masks/Normal-233.png,True,G11086,test,ec0777ecd03ceaf8825690a0450957db,d02dc63a5f6d1c4c
Normal-2330.png,Normal,Normal/images/Normal-2330.png,Normal/masks/Normal-2330.png,True,G11087,train,dc8e39537a86a3c1b795694b598294e6,9879851e1d451f3d
Normal-2331.png,Normal,Normal/images/Normal-2331.png,Normal/masks/Normal-2331.png,True,G11088,train,718a84708ab55a730661fce5e6e1ade5,dc2c420f0fecadac
Normal-2332.png,Normal,Normal/images/Normal-2332.png,Normal/masks/Normal-2332.png,True,G11089,train,6f520744edf8773191889d1762727652,c12d4a1e191f7d39
Normal-2333.png,Normal,Normal/images/Normal-2333.png,Normal/masks/Normal-2333.png,True,G11090,train,68edbf1af399f42bb8b48d768c613898,c637d849d068db6a
Normal-2334.png,Normal,Normal/images/Normal-2334.png,Normal/masks/Normal-2334.png,True,G11091,train,fbb7a5494895e3f27d51644ee735ae50,cdae825bb4a2b0f2
Normal-2335.png,Normal,Normal/images/Normal-2335.png,Normal/masks/Normal-2335.png,True,G11092,train,c791e2962bb8847c76fffd8e864f5c37,9a79851e851e1bc7
Normal-2336.png,Normal,Normal/images/Normal-2336.png,Normal/masks/Normal-2336.png,True,G11093,train,3929084464b17536f07b28e653c1ad66,9168c6961a194f7f
Normal-2337.png,Normal,Normal/images/Normal-2337.png,Normal/masks/Normal-2337.png,True,G11094,train,c9ae1cf4b449bd4ac8630c7d89275bd8,dd2a531d5c19191f
Normal-2338.png,Normal,Normal/images/Normal-2338.png,Normal/masks/Normal-2338.png,True,G11095,train,e9eb2c1944139bba6910b97026119671,887b05165a4d5d7b
Normal-2339.png,Normal,Normal/images/Normal-2339.png,Normal/masks/Normal-2339.png,True,G11096,val,87abca4a0a6d94e22a07a94fafb9371a,9179861e185e5e79
Normal-234.png,Normal,Normal/images/Normal-234.png,Normal/masks/Normal-234.png,True,G11097,train,522dd59e8529cec32171b30c6c02570f,947da73e8d2d180d
Normal-2340.png,Normal,Normal/images/Normal-2340.png,Normal/masks/Normal-2340.png,True,G11098,train,c4e94356fc4f20df4ec279ca765a5213,dd2c531343421f7b
Normal-2341.png,Normal,Normal/images/Normal-2341.png,Normal/masks/Normal-2341.png,True,G11099,train,e18bc332af0055642f61f2f9f16c948c,cc0c434f1c6c7d3e
Normal-2342.png,Normal,Normal/images/Normal-2342.png,Normal/masks/Normal-2342.png,True,G11100,train,b102858705feee1f73b166336a53d958,c17a12421f2d373f
Normal-2343.png,Normal,Normal/images/Normal-2343.png,Normal/masks/Normal-2343.png,True,G11101,train,349982a9731ed9ac8ace8ededed7b204,dd3c43421d4bad3c
Normal-2344.png,Normal,Normal/images/Normal-2344.png,Normal/masks/Normal-2344.png,True,G11102,val,91c682bbc7204d518405f79690307468,dd29f1075ec40e1e
Normal-2345.png,Normal,Normal/images/Normal-2345.png,Normal/masks/Normal-2345.png,True,G11103,val,139da33e013a0f73380722d8c443952f,c50fd7436922323f
Normal-2346.png,Normal,Normal/images/Normal-2346.png,Normal/masks/Normal-2346.png,True,G11104,test,fe9531239b9868e0ebf189e8796d2a6e,dc2ed2d30aca4b1b
Normal-2347.png,Normal,Normal/images/Normal-2347.png,Normal/masks/Normal-2347.png,True,G11105,test,c953cd9ea6e9c2208fed958b2eb5edce,fd2cd31b55471944
Normal-2348.png,Normal,Normal/images/Normal-2348.png,Normal/masks/Normal-2348.png,True,G11106,train,04e9f3cb3d9bb0495a5528337c099b26,c17b4616194b7f34
Normal-2349.png,Normal,Normal/images/Normal-2349.png,Normal/masks/Normal-2349.png,True,G11107,test,9c558fd2094301c851bad9930bec5022,8979875efa09f828
Normal-235.png,Normal,Normal/images/Normal-235.png,Normal/masks/Normal-235.png,True,G11108,train,6296ea7c073ce6511e67973c8d416f20,d0fec73a842e32c6
Normal-2350.png,Normal,Normal/images/Normal-2350.png,Normal/masks/Normal-2350.png,True,G11109,train,12fa5960f0c3da6a3ed04613b99fb1c9,dc2d42534f4e2d2d
Normal-2351.png,Normal,Normal/images/Normal-2351.png,Normal/masks/Normal-2351.png,True,G11110,train,ffa1ee747b43c330b9a8c4f0fd0e8ccc,c83c471b095c7d79
Normal-2352.png,Normal,Normal/images/Normal-2352.png,Normal/masks/Normal-2352.png,True,G11111,train,4133d701fee50c88ed6f5188ae7d234d,dd0ac30f4f2c3c3c
Normal-2353.png,Normal,Normal/images/Normal-2353.png,Normal/masks/Normal-2353.png,True,G11112,train,89fd8dc09008ca072ec7bad7a9180f61,dd28d60b5d0a79e1
Normal-2354.png,Normal,Normal/images/Normal-2354.png,Normal/masks/Normal-2354.png,True,G11113,train,a81aa5df1fbf1615ae653bc0751c6c73,d42d425f0b4f0f3c
Normal-2355.png,Normal,Normal/images/Normal-2355.png,Normal/masks/Normal-2355.png,True,G11114,train,002c8bebdec253ef6756aad1fa11c364,dd2f025f164a1f49
Normal-2356.png,Normal,Normal/images/Normal-2356.png,Normal/masks/Normal-2356.png,True,G11115,train,8223021498f2107a6e3d07dc4ead9c88,dc6cd30b0f481f8d
Normal-2357.png,Normal,Normal/images/Normal-2357.png,Normal/masks/Normal-2357.png,True,G11116,train,501f5440d9e6c658e215b542af5057e1,9979071e1ec55e19
Normal-2358.png,Normal,Normal/images/Normal-2358.png,Normal/masks/Normal-2358.png,True,G11117,train,8c7a08d8c7820ccc87f76c3add49b8d4,d03c6fc6505bc24f
Normal-2359.png,Normal,Normal/images/Normal-2359.png,Normal/masks/Normal-2359.png,True,G11118,val,d7f649442e78c644916dd3f6144c92d0,8129de5c585e7879
Normal-236.png,Normal,Normal/images/Normal-236.png,Normal/masks/Normal-236.png,True,G11119,train,b862e9b2817deae9b5ba7166fb3a69f5,d47ec75b8c852cc4
Normal-2360.png,Normal,Normal/images/Normal-2360.png,Normal/masks/Normal-2360.png,True,G11120,test,dc82d48e00df11f9b5e0ee7db0d240b3,d03d430ace6e7c6c
Normal-2361.png,Normal,Normal/images/Normal-2361.png,Normal/masks/Normal-2361.png,True,G11121,train,9ea2d3ab52959e5684e1e6e829462c9f,916986161bd75e1b
Normal-2362.png,Normal,Normal/images/Normal-2362.png,Normal/masks/Normal-2362.png,True,G11122,train,e99535e85495257c7713997eb1ada7cc,d93a52131a4d1fbc
Normal-2363.png,Normal,Normal/images/Normal-2363.png,Normal/masks/Normal-2363.png,True,G11123,train,2db53ffe7a23317f3c3e0bf892069f00,993dc6161b974e19
Normal-2364.png,Normal,Normal/images/Normal-2364.png,Normal/masks/Normal-2364.png,True,G11124,val,761db72c400e2bb0caf1707339c807cd,d979c61e18491b1f
Normal-2365.png,Normal,Normal/images/Normal-2365.png,Normal/masks/Normal-2365.png,True,G11125,val,1ef7e721dbceff9f9aa1670bf882d1e2,817f871a5d6838e5
Normal-2366.png,Normal,Normal/images/Normal-2366.png,Normal/masks/Normal-2366.png,True,G11126,test,3dc0e55996cc3bb4ac9e202fa0570ad1,dc7f434b090c2b3d
Normal-2367.png,Normal,Normal/images/Normal-2367.png,Normal/masks/Normal-2367.png,True,G11127,train,968afd49c25cbcd57dfd8e94759f0b0d,997d86121fc61e1d
Normal-2368.png,Normal,Normal/images/Normal-2368.png,Normal/masks/Normal-2368.png,True,G11128,train,c4061e9f96a06e580fe92f92c7fa49e3,9c7c430b892f3cbc
Normal-2369.png,Normal,Normal/images/Normal-2369.png,Normal/masks/Normal-2369.png,True,G11129,train,92dd53fce3da18451e4cfc454a710c4e,d97946561c696d38
Normal-237.png,Normal,Normal/images/Normal-237.png,Normal/masks/Normal-237.png,True,G11130,train,435c86f6b54a20f12e8e012b359c2fd3,902dcf3e9c491dc9
Normal-2370.png,Normal,Normal/images/Normal-2370.png,Normal/masks/Normal-2370.png,True,G11131,train,ab9632e794a7678fc72d457143e9cd9a,993dc65b455b1d44
Normal-2371.png,Normal,Normal/images/Normal-2371.png,Normal/masks/Normal-2371.png,True,G11132,train,933e2be7766f3932241f1f0f1b9ab104,9139575714524f1f
Normal-2372.png,Normal,Normal/images/Normal-2372.png,Normal/masks/Normal-2372.png,True,G11133,train,8ff6d35b016e0a516d097ede07868bd9,dd2cd34b0c0e2b3d
Normal-2373.png,Normal,Normal/images/Normal-2373.png,Normal/masks/Normal-2373.png,True,G11134,train,80dc47b60a21e4ffaade33463d9f1ee9,c97b441a57451a5f
Normal-2374.png,Normal,Normal/images/Normal-2374.png,Normal/masks/Normal-2374.png,True,G11135,train,271d3c9dcb984b2b60266d90d95d23f6,bd2c431b5e0b3d4a
Normal-2375.png,Normal,Normal/images/Normal-2375.png,Normal/masks/Normal-2375.png,True,G11136,train,c33c4f5b582d634de2539fd05a7403b6,dd3bd31f420b0c2b
Normal-2376.png,Normal,Normal/images/Normal-2376.png,Normal/masks/Normal-2376.png,True,G11137,train,c717bc9def7dca4a128d291400339351,c82957861e574f59
Normal-2377.png,Normal,Normal/images/Normal-2377.png,Normal/masks/Normal-2377.png,True,G11138,val,f6f0bdd87b8cdb15aa7dbcd2ff4cf666,cc3d03434f4b4f1d
Normal-2378.png,Normal,Normal/images/Normal-2378.png,Normal/masks/Normal-2378.png,True,G11139,val,6ec8ffbdd0da64e1e2a8d11b405c0d8a,997b86121f4f1a1d
Normal-2379.png,Normal,Normal/images/Normal-2379.png,Normal/masks/Normal-2379.png,True,G11140,train,9e836a5420f91b4b237d6354ab72284c,d82fc35f0c442f5c
Normal-238.png,Normal,Normal/images/Normal-238.png,Normal/masks/Normal-238.png,True,G11141,test,002d6677c1d5fbbe25da940aec1d14c0,913bd25a98f131b5
Normal-2380.png,Normal,Normal/images/Normal-2380.png,Normal/masks/Normal-2380.png,True,G11142,test,5ec441e9610a1df9123ce5786e97e9ca,dd0e061e0a6f7d68
Normal-2381.png,Normal,Normal/images/Normal-2381.png,Normal/masks/Normal-2381.png,True,G11143,train,553dabd984624b602dd7523eec03a969,dc2cc31b4d4e5c4d
Normal-2382.png,Normal,Normal/images/Normal-2382.png,Normal/masks/Normal-2382.png,True,G11144,train,b045146635a0968ea7caefc16738ba1e,dc2d425f4e4c6d2c
Normal-2383.png,Normal,Normal/images/Normal-2383.png,Normal/masks/Normal-2383.png,True,G11145,train,4e184b990606c33081cdb93dc48faef4,9968071a4d5b59ed
Normal-2384.png,Normal,Normal/images/Normal-2384.png,Normal/masks/Normal-2384.png,True,G11146,train,b287d598bb0bc75cf437774e1cfe16d3,d97cc71b1c485b1c
Normal-2385.png,Normal,Normal/images/Normal-2385.png,Normal/masks/Normal-2385.png,True,G11147,train,c7a82880577c5fd296b4bfc5a0dc7809,9c2ec70b446d38f5
Normal-2386.png,Normal,Normal/images/Normal-2386.png,Normal/masks/Normal-2386.png,True,G11148,val,ab47c618fe7e551b0cc0a8c6968e9c36,bd6dc21a4d481b6b
Normal-2387.png,Normal,Normal/images/Normal-2387.png,Normal/masks/Normal-2387.png,True,G11149,train,21c1ca09d2b9d6ad8aff94b89efbd8dd,992d061a8f1e1b7d
Normal-2388.png,Normal,Normal/images/Normal-2388.png,Normal/masks/Normal-2388.png,True,G11150,test,07faf9966385f60f8e08a9db2f53d4bb,8978971a195d78d9
Normal-2389.png,Normal,Normal/images/Normal-2389.png,Normal/masks/Normal-2389.png,True,G11151,train,d1834d59e8c5a7d342b58ea009d28180,d968861a1e4d5f39
Normal-239.png,Normal,Normal/images/Normal-239.png,Normal/masks/Normal-239.png,True,G11152,train,06e458b6b64d49b33efeacfa3ff6125b,d074c63ab92d3cc6
Normal-2390.png,Normal,Normal/images/Normal-2390.png,Normal/masks/Normal-2390.png,True,G11153,train,e1e5bd654b3d3fea7d0bee080cc2165c,997c06191f6e39e1
Normal-2391.png,Normal,Normal/images/Normal-2391.png,Normal/masks/Normal-2391.png,True,G11154,train,2d1717c937108ab151250807c2cb1b84,bc4c431b4d4d1ccf
Normal-2392.png,Normal,Normal/images/Normal-2392.png,Normal/masks/Normal-2392.png,True,G11155,train,b89ac3adde2196e236ce1a066d0604c0,9849c61a493f36e7
Normal-2393.png,Normal,Normal/images/Normal-2393.png,Normal/masks/Normal-2393.png,True,G11156,val,6c611d39807732d4a4480a6965d1b9cf,cc3cc35f534c1f90
Normal-2394.png,Normal,Normal/images/Normal-2394.png,Normal/masks/Normal-2394.png,True,G11157,train,d52d0e08e8487135e2d73752d7d81bbb,997c861a594d5e4b
Normal-2395.png,Normal,Normal/images/Normal-2395.png,Normal/masks/Normal-2395.png,True,G11158,train,86dc432c28374551bcf3246e1f4c01d5,dd2dd25f84471964
Normal-2396.png,Normal,Normal/images/Normal-2396.png,Normal/masks/Normal-2396.png,True,G11159,train,72471988ae86138a4a5dd838db93e494,d81b0716185b6b7b
Normal-2397.png,Normal,Normal/images/Normal-2397.png,Normal/masks/Normal-2397.png,True,G11160,train,651e52690776c7c8d7d75daa3c2cb9f4,d97dc6561c4c4d1a
Normal-2398.png,Normal,Normal/images/Normal-2398.png,Normal/masks/Normal-2398.png,True,G11161,val,ec715fdff172dcb4cc4493b4a5f87321,d42e03135b475f5c
Normal-2399.png,Normal,Normal/images/Normal-2399.png,Normal/masks/Normal-2399.png,True,G11162,val,bcbf995cc3cb447b9685c66b2fbefbad,d92f461919e4b9b8
Normal-24.png,Normal,Normal/images/Normal-24.png,Normal/masks/Normal-24.png,True,G11163,test,7a8e2558e60df49eec2c14dc7b6fd398,dc3dc31b11473d19
Normal-240.png,Normal,Normal/images/Normal-240.png,Normal/masks/Normal-240.png,True,G11164,test,63a9533bd387bff4e8d6039156ee1f4e,d46ec75bcd6428e0
Normal-2400.png,Normal,Normal/images/Normal-2400.png,Normal/masks/Normal-2400.png,True,G11165,test,00c5cd7b464859ceab6e67193efb904a,c13916c79a1cedc9
Normal-2401.png,Normal,Normal/images/Normal-2401.png,Normal/masks/Normal-2401.png,True,G11166,train,27c8f7bdca4ba61e8fd0eecb57622938,817c4e35c5cb39a6
Normal-2402.png,Normal,Normal/images/Normal-2402.png,Normal/masks/Normal-2402.png,True,G11167,train,062c2b5a646aa0beb4fab532b588819a,dc2ed30b424f0f0f
Normal-2403.png,Normal,Normal/images/Normal-2403.png,Normal/masks/Normal-2403.png,True,G11168,train,2ed9e235d0b662961e9c7023082f41c9,dd2d530b1b4a3f18
Normal-2404.png,Normal,Normal/images/Normal-2404.png,Normal/masks/Normal-2404.png,True,G11169,train,508fe4fa441609884ba4f05597800d54,c979061e1d4b3dc9
Normal-2405.png,Normal,Normal/images/Normal-2405.png,Normal/masks/Normal-2405.png,True,G11170,train,7e139d4305cf45798363adaf14e8813e,993fc61a1d19594d
Normal-2406.png,Normal,Normal/images/Normal-2406.png,Normal/masks/Normal-2406.png,True,G11171,train,b10596b0acf32cc244221fd8fb582c5a,d97b461a0e0f3b2c
Normal-2407.png,Normal,Normal/images/Normal-2407.png,Normal/masks/Normal-2407.png,True,G11172,train,d06d94fe3647b70d4053630f62ae842e,9169ce16111f5e5b
Normal-2408.png,Normal,Normal/images/Normal-2408.png,Normal/masks/Normal-2408.png,True,G11173,test,d09282133f7108d60873d378c358ad8e,9978871a1d4d1b79
Normal-2409.png,Normal,Normal/images/Normal-2409.png,Normal/masks/Normal-2409.png,True,G11174,train,89cfd4053f1af1f5e1cc7f7a441bce2e,81785647191f4e3f
Normal-241.png,Normal,Normal/images/Normal-241.png,Normal/masks/Normal-241.png,True,G11175,train,2cec6e1028e5580e8b242809856b8441,d46ec319ccecb8c2
Normal-2410.png,Normal,Normal/images/Normal-2410.png,Normal/masks/Normal-2410.png,True,G11176,train,b2b962f14b916a2fcebba2b54ae1bd61,987f561a591b292d
Normal-2411.png,Normal,Normal/images/Normal-2411.png,Normal/masks/Normal-2411.png,True,G11177,train,7723e2225c2d1a534a1220ca1780a3e6,c979c65a1c5a1d59
Normal-2412.png,Normal,Normal/images/Normal-2412.png,Normal/masks/Normal-2412.png,True,G11178,train,87b3f5b4b14981f7eaf4b1928c791a17,dc2d42570b0e6d1f
Normal-2413.png,Normal,Normal/images/Normal-2413.png,Normal/masks/Normal-2413.png,True,G11179,train,7049a26cea7afa8665e3aec87a47a7a5,dc2dd2431f424f1d
Normal-2414.png,Normal,Normal/images/Normal-2414.png,Normal/masks/Normal-2414.png,True,G11180,test,eacff90c79c14c78bdd7829aac4f1c82,dc2ed30b1da4383d
Normal-2415.png,Normal,Normal/images/Normal-2415.png,Normal/masks/Normal-2415.png,True,G11181,val,fc2e547c1a14f0918e8c95f7c4b126c3,d139c206145adf6f
Normal-2416.png,Normal,Normal/images/Normal-2416.png,Normal/masks/Normal-2416.png,True,G11182,test,e00f7870e036b372cde1fe0a814869f4,887f971a052a5f4d
Normal-2417.png,Normal,Normal/images/Normal-2417.png,Normal/masks/Normal-2417.png,True,G11183,val,87877f4cd784455f706ec6c4f08e2c7a,993fc4561e4c5b19
Normal-2418.png,Normal,Normal/images/Normal-2418.png,Normal/masks/Normal-2418.png,True,G11184,val,5a24f65203daf521d7619bd99958cbba,9879c61e18491f5f
Normal-2419.png,Normal,Normal/images/Normal-2419.png,Normal/masks/Normal-2419.png,True,G11185,train,423a4c51a8df2ba9a12b590016aeccdc,dc2ec35b086c3dac
Normal-242.png,Normal,Normal/images/Normal-242.png,Normal/masks/Normal-242.png,True,G11186,train,28575980c7586d34a1b7d61f2f31702d,8075875ec82d6dc7
Normal-2420.png,Normal,Normal/images/Normal-2420.png,Normal/masks/Normal-2420.png,True,G11187,train,8b927e010956bce3660005b073ab51d5,cc3d42d24f4eac2d
Normal-2421.png,Normal,Normal/images/Normal-2421.png,Normal/masks/Normal-2421.png,True,G11188,train,33bc27443994922bde49970b31fbaa52,8968165b5b5d79a1
Normal-2422.png,Normal,Normal/images/Normal-2422.png,Normal/masks/Normal-2422.png,True,G11189,train,c091182035d17d944b7a496b3b973003,dd7cc35b460b0d0e
Normal-2423.png,Normal,Normal/images/Normal-2423.png,Normal/masks/Normal-2423.png,True,G11190,train,44efbd4c1330cbfa85c02e3a881b07b1,ddaf0615c71c414f
Normal-2424.png,Normal,Normal/images/Normal-2424.png,Normal/masks/Normal-2424.png,True,G11191,train,f9a464eaf838f58e1872b1819ed27664,897d061a457eb8e9
Normal-2425.png,Normal,Normal/images/Normal-2425.png,Normal/masks/Normal-2425.png,True,G11192,train,d8ed6ace01c073bc671ccba8ce4fefd0,c03d42831f1e6d3f
Normal-2426.png,Normal,Normal/images/Normal-2426.png,Normal/masks/Normal-2426.png,True,G11193,train,b074170735459c8620778f1b32ba6b0a,dd28071b494e787d
Normal-2427.png,Normal,Normal/images/Normal-2427.png,Normal/masks/Normal-2427.png,True,G11194,train,dda941fd70d3d9c489a1546b2354058a,982d071e051f5d5f
Normal-2428.png,Normal,Normal/images/Normal-2428.png,Normal/masks/Normal-2428.png,True,G11195,train,0e7df4b5abcc6f12e095bcabbc697950,dd2e430b544f1c4f
Normal-2429.png,Normal,Normal/images/Normal-2429.png,Normal/masks/Normal-2429.png,True,G11196,train,1a0d5a5d3a044555f56c2ca565779079,813b86061c5f5f3e
Normal-243.png,Normal,Normal/images/Normal-243.png,Normal/masks/Normal-243.png,True,G11197,val,8bea429db6a3cc3164430cba1ab25a77,813aee189c7dbcc4
Normal-2430.png,Normal,Normal/images/Normal-2430.png,Normal/masks/Normal-2430.png,True,G11198,train,0a38d77a84fca505a01fc035c9c8ca24,dd2a524a5e4c617d
Normal-2431.png,Normal,Normal/images/Normal-2431.png,Normal/masks/Normal-2431.png,True,G11199,train,d096c9605b6c15a2ea871ac578c0952b,997d861a59055bb9
Normal-2432.png,Normal,Normal/images/Normal-2432.png,Normal/masks/Normal-2432.png,True,G11200,val,4018373cba7148b4b868ee5eb5789a36,f969879605c71d85
Normal-2433.png,Normal,Normal/images/Normal-2433.png,Normal/masks/Normal-2433.png,True,G11201,test,a0033f01523a976adb7d7912190f2a06,dc3dc20d144f1e5d
Normal-2434.png,Normal,Normal/images/Normal-2434.png,Normal/masks/Normal-2434.png,True,G11202,val,811629dc839b8b764890ca88a63d928d,dd2d471b454d1d45
Normal-2435.png,Normal,Normal/images/Normal-2435.png,Normal/masks/Normal-2435.png,True,G11203,test,39852b2ec62b8adcc3012788962fb1c3,817a161a1d5f1b1f
Normal-2436.png,Normal,Normal/images/Normal-2436.png,Normal/masks/Normal-2436.png,True,G11204,train,858430a5bd6683b564620f5362e2564a,c97b56825b192b1d
Normal-2437.png,Normal,Normal/images/Normal-2437.png,Normal/masks/Normal-2437.png,True,G11205,train,f8719e60bf8cbed618dc4aa89cb64621,d4a9d2435a534b5e
Normal-2438.png,Normal,Normal/images/Normal-2438.png,Normal/masks/Normal-2438.png,True,G11206,test,d90f5c2c0f96976881590f29694c8f71,9968861b1c4e5ff8
Normal-2439.png,Normal,Normal/images/Normal-2439.png,Normal/masks/Normal-2439.png,True,G11207,train,44737d31eb067fba49fe1c87cb9cdd36,dc4a530b4d3e0b2f
Normal-244.png,Normal,Normal/images/Normal-244.png,Normal/masks/Normal-244.png,True,G11208,train,005b203df11b12f4910e6045bfdcf8ac,85f9e75adac55840
Normal-2440.png,Normal,Normal/images/Normal-2440.png,Normal/masks/Normal-2440.png,True,G11209,train,1be478bcce57c7d0948d61fefd8e3b13,d939061a1a4e7b79
Normal-2441.png,Normal,Normal/images/Normal-2441.png,Normal/masks/Normal-2441.png,True,G11210,train,5497570e8713e9b684d4a708c5e2cd5b,dd2856c71b1f4453
Normal-2442.png,Normal,Normal/images/Normal-2442.png,Normal/masks/Normal-2442.png,True,G11211,test,2b3336475d95addd52a87a789287728e,802957d71b5c6dd8
Normal-2443.png,Normal,Normal/images/Normal-2443.png,Normal/masks/Normal-2443.png,True,G11212,train,a98259f2f0a29af9814590df6c801c86,817996065a1d7d4f
Normal-2444.png,Normal,Normal/images/Normal-2444.png,Normal/masks/Normal-2444.png,True,G11213,train,db5695d14321bd526f2ea7a0fb2c96c9,ec2cd30b0d4a5f4e
Normal-2445.png,Normal,Normal/images/Normal-2445.png,Normal/masks/Normal-2445.png,True,G11214,train,78eaf43ddc3978130fde8db375fcc0b3,d939c61f16862b9a
Normal-2446.png,Normal,Normal/images/Normal-2446.png,Normal/masks/Normal-2446.png,True,G11215,train,f54c03f56f25bead7d49ca9b6da73ca2,887d47c71b1c2cec
Normal-2447.png,Normal,Normal/images/Normal-2447.png,Normal/masks/Normal-2447.png,True,G11216,train,1bbfb14d6f44d942fc6e74754d5da279,995f161a5b7aa138
Normal-2448.png,Normal,Normal/images/Normal-2448.png,Normal/masks/Normal-2448.png,True,G11217,train,e3b3252d9c540d130090a0780fea1d52,d82cc65b198c6abd
Normal-2449.png,Normal,Normal/images/Normal-2449.png,Normal/masks/Normal-2449.png,True,G11218,val,c6f55bc68c2d87efd9b6c251c3ac1776,d92dc61b1d4a1b4d
Normal-245.png,Normal,Normal/images/Normal-245.png,Normal/masks/Normal-245.png,True,G11219,train,179cb178383006492ab68d0c18149f58,913bd65a585919da
Normal-2450.png,Normal,Normal/images/Normal-2450.png,Normal/masks/Normal-2450.png,True,G11220,train,1dc42df30d43721f245f8ccf7a5d6f10,987f851a1f45431f
Normal-2451.png,Normal,Normal/images/Normal-2451.png,Normal/masks/Normal-2451.png,True,G11221,train,e84e55b62a61f4fe1cc1b259fb20cbfe,987d4e1b79c49293
Normal-2452.png,Normal,Normal/images/Normal-2452.png,Normal/masks/Normal-2452.png,True,G11222,val,050f7fe242ff57a8af0d88a7dd73075d,d43dd6d24b484e4b
Normal-2453.png,Normal,Normal/images/Normal-2453.png,Normal/masks/Normal-2453.png,True,G11223,test,ab910d739fc5a24b0b8b92cbd92f0db5,dc69821f0d4f1d59
Normal-2454.png,Normal,Normal/images/Normal-2454.png,Normal/masks/Normal-2454.png,True,G11224,val,41975061aa0377aa5a3bf24adaea4e10,993d061e1c5f1b1d
Normal-2455.png,Normal,Normal/images/Normal-2455.png,Normal/masks/Normal-2455.png,True,G11225,test,e7dc630f2b80001ff50d3db99462cdbd,8978871a195f58f9
Normal-2456.png,Normal,Normal/images/Normal-2456.png,Normal/masks/Normal-2456.png,True,G11226,train,cf790c93182e7299095bb8dc71462420,d87be5561e415b18
Normal-2457.png,Normal,Normal/images/Normal-2457.png,Normal/masks/Normal-2457.png,True,G11227,train,b245d1627d5747fb1b6a8ce0b4fec85c,c63f41490eeb2d2e
Normal-2458.png,Normal,Normal/images/Normal-2458.png,Normal/masks/Normal-2458.png,True,G11228,train,e0b7493a9e90ec437389a259d79499e0,dd2cd25b0e4e1e0e
Normal-2459.png,Normal,Normal/images/Normal-2459.png,Normal/masks/Normal-2459.png,True,G11229,train,993c28c98371d05fb9addbeb4e328c7f,917f06561a596979
Normal-246.png,Normal,Normal/images/Normal-246.png,Normal/masks/Normal-246.png,True,G11230,train,76b4b147a2461e52a006cb2e9ea0f1c6,d03ad21f1c551d9d
Normal-2460.png,Normal,Normal/images/Normal-2460.png,Normal/masks/Normal-2460.png,True,G11231,train,1a9dab38f8e83d2ce0b1238f818dbe5f,882b85561f175f1e
Normal-2461.png,Normal,Normal/images/Normal-2461.png,Normal/masks/Normal-2461.png,True,G11232,train,d2484633bf5bef9e9db3e6873ec234c8,dd28c50b1f08ef1e
Normal-2462.png,Normal,Normal/images/Normal-2462.png,Normal/masks/Normal-2462.png,True,G11233,train,e1e2bc5d7576304e393aedb339dc3f43,987b8516151f1e5d
Normal-2463.png,Normal,Normal/images/Normal-2463.png,Normal/masks/Normal-2463.png,True,G11234,train,6f351d4dbea421d56b16f9d9cb58adc2,d47e42434b5b0f4d
Normal-2464.png,Normal,Normal/images/Normal-2464.png,Normal/masks/Normal-2464.png,True,G11235,train,0870531bef19dd5380a64550d1590c9f,8979061e5e18fe59
Normal-2465.png,Normal,Normal/images/Normal-2465.png,Normal/masks/Normal-2465.png,True,G11236,train,819608c1a510787c628ca233ba39a833,d92fc68219d5c61d
Normal-2466.png,Normal,Normal/images/Normal-2466.png,Normal/masks/Normal-2466.png,True,G11237,train,6f32152a838c6a39f1eae535ac8e0eae,916906165a9f7a79
Normal-2467.png,Normal,Normal/images/Normal-2467.png,Normal/masks/Normal-2467.png,True,G11238,train,e2c910098cb2c095917ac25dbe77ac6a,990d061e1a4f5e7e
Normal-2468.png,Normal,Normal/images/Normal-2468.png,Normal/masks/Normal-2468.png,True,G11239,val,93bd94109a8c68bd62ba19462cff9a71,dd2cc6d25b450e5c
Normal-2469.png,Normal,Normal/images/Normal-2469.png,Normal/masks/Normal-2469.png,True,G11240,train,360e57007e07e443deb78e027d9c5af2,dd28d20b470f4f4d
Normal-247.png,Normal,Normal/images/Normal-247.png,Normal/masks/Normal-247.png,True,G11241,val,15f5b9fee959aa32512cf917d9a68dba,916c863bad3d6887
Normal-2470.png,Normal,Normal/images/Normal-2470.png,Normal/masks/Normal-2470.png,True,G11242,val,631c0ed4c9867dc60b9422afd657ad21,905e061e5a5f3aea
Normal-2471.png,Normal,Normal/images/Normal-2471.png,Normal/masks/Normal-2471.png,True,G11243,test,ab75054cac5b20dcff784a2c065e7b25,916886161b5f797a
Normal-2472.png,Normal,Normal/images/Normal-2472.png,Normal/masks/Normal-2472.png,True,G11244,test,ea5d05aae804cfe888f962d32ba5b6c0,8129c61e154f5f3b
Normal-2473.png,Normal,Normal/images/Normal-2473.png,Normal/masks/Normal-2473.png,True,G11245,val,6ac88f70e1057578b0de192671bae7e7,d42e024b3cececb9
Normal-2474.png,Normal,Normal/images/Normal-2474.png,Normal/masks/Normal-2474.png,True,G11246,test,178712f9bd600de6c367e933de9cd67c,897986961e1e5f19
Normal-2475.png,Normal,Normal/images/Normal-2475.png,Normal/masks/Normal-2475.png,True,G11247,train,1615b0f9a0a85533e152009397baaae0,cc3c03439f4abc5d
Normal-2476.png,Normal,Normal/images/Normal-2476.png,Normal/masks/Normal-2476.png,True,G11248,train,54e08e875fc33627c70e31258a918d1c,c4dc6cc70f2acd4c
Normal-2477.png,Normal,Normal/images/Normal-2477.png,Normal/masks/Normal-2477.png,True,G11249,train,5d0cdeb3e467551fb1eb5b89cb25a169,d96f061a0e1aad79
Normal-2478.png,Normal,Normal/images/Normal-2478.png,Normal/masks/Normal-2478.png,True,G11250,train,6b31b3e5436dc9e21d2b7d7ee39d7bb3,887ba5c71696c547
Normal-2479.png,Normal,Normal/images/Normal-2479.png,Normal/masks/Normal-2479.png,True,G11251,test,dca3a198d60439908512e8a34e48967a,9929461e5a497b79
Normal-248.png,Normal,Normal/images/Normal-248.png,Normal/masks/Normal-248.png,True,G11252,train,ec7421e5586bb85dc6f45f19ffbf55a6,957e8f1a095c5ac6
Normal-2480.png,Normal,Normal/images/Normal-2480.png,Normal/masks/Normal-2480.png,True,G11253,train,033ca28805e5fa612da7ced4f1ae3016,8039c61e1f7f3919
Normal-2481.png,Normal,Normal/images/Normal-2481.png,Normal/masks/Normal-2481.png,True,G11254,val,ec5abfd4ceb41e2dbe90406409d495da,d12f02075e5b7d58
Normal-2482.png,Normal,Normal/images/Normal-2482.png,Normal/masks/Normal-2482.png,True,G11255,train,dd6faed8ce77f17ecb8d6ec0937643a6,8d0dc61a534f1acf
Normal-2483.png,Normal,Normal/images/Normal-2483.png,Normal/masks/Normal-2483.png,True,G11256,train,19324c62514d2013d5cbc7b0378d260e,f979d61e074a184b
Normal-2484.png,Normal,Normal/images/Normal-2484.png,Normal/masks/Normal-2484.png,True,G11257,train,2cb1648f43a34defad51c8ac0ca7c2d2,887d061f471e594f
Normal-2485.png,Normal,Normal/images/Normal-2485.png,Normal/masks/Normal-2485.png,True,G11258,val,e14747e8f9d70007e7efb1f8968b7e4a,887f475a134d1f49
Normal-2486.png,Normal,Normal/images/Normal-2486.png,Normal/masks/Normal-2486.png,True,G11259,train,57172862c2b0a1517e3dc2dbd1a099ac,dd2d825f444e1f4c
Normal-2487.png,Normal,Normal/images/Normal-2487.png,Normal/masks/Normal-2487.png,True,G11260,test,4d516a32f5e71a2c85165dd25a2d3081,d02e475f1a0e3e3a
Normal-2488.png,Normal,Normal/images/Normal-2488.png,Normal/masks/Normal-2488.png,True,G11261,train,1d6eda6dd7e6f73b5cf9617cf7d36b0f,a97f061e19195b69
Normal-2489.png,Normal,Normal/images/Normal-2489.png,Normal/masks/Normal-2489.png,True,G11262,test,cf84316bd3eef7075355307ceabc6870,817ac616184f5f3b
Normal-249.png,Normal,Normal/images/Normal-249.png,Normal/masks/Normal-249.png,True,G11263,train,f77f211260f99bc2abb5e70a8e43d56c,817f0f1a0d5d594d
Normal-2490.png,Normal,Normal/images/Normal-2490.png,Normal/masks/Normal-2490.png,True,G11264,test,1ce6c94b587d9a7cd35f81079a375189,cc7d405b4b4d1e2b
Normal-2491.png,Normal,Normal/images/Normal-2491.png,Normal/masks/Normal-2491.png,True,G11265,train,b8e6cc223e8b7a34a415a2de3bcef531,dd3d464b194c4a3d
Normal-2492.png,Normal,Normal/images/Normal-2492.png,Normal/masks/Normal-2492.png,True,G11266,train,22809fd14c5e79d15d9fdcbde14de192,dd3c47431a486d3e
Normal-2493.png,Normal,Normal/images/Normal-2493.png,Normal/masks/Normal-2493.png,True,G11267,train,dea65ebd8a0e171b6713da7a7b170840,81e8e69658ed7938
Normal-2494.png,Normal,Normal/images/Normal-2494.png,Normal/masks/Normal-2494.png,True,G11268,train,a1ca7ccb3dc6d64b79934bf0e2a9d451,dc2dc21b1d4f0f1c
Normal-2495.png,Normal,Normal/images/Normal-2495.png,Normal/masks/Normal-2495.png,True,G11269,train,d08a9cfc60470ff0f7e275bd257eee4a,dd2ed2130f4d183d
Normal-2496.png,Normal,Normal/images/Normal-2496.png,Normal/masks/Normal-2496.png,True,G11270,train,c9e141faef0dda932aae58126541818a,cc2fc30b0f2a2dae
Normal-2497.png,Normal,Normal/images/Normal-2497.png,Normal/masks/Normal-2497.png,True,G11271,test,5ce185619edae4535e39f33e85b31491,996dc712951eba07
Normal-2498.png,Normal,Normal/images/Normal-2498.png,Normal/masks/Normal-2498.png,True,G11272,train,c82abb4f7942516fec1d47f0268f2846,887f030f1e1b1d2f
Normal-2499.png,Normal,Normal/images/Normal-2499.png,Normal/masks/Normal-2499.png,True,G11273,train,398a9b67b64fa83b821be3fd83042f6f,9d61eece1a1a1a0f
Normal-25.png,Normal,Normal/images/Normal-25.png,Normal/masks/Normal-25.png,True,G11274,train,8971919170b3d1293147c06469d67799,9919861a194f7bb9
Normal-250.png,Normal,Normal/images/Normal-250.png,Normal/masks/Normal-250.png,True,G11275,train,a5c0dcc3e143b101b979f16ab9630d9e,d1e4c729cd6d09c6
Normal-2500.png,Normal,Normal/images/Normal-2500.png,Normal/masks/Normal-2500.png,True,G11276,train,d0ca59227b34dbe9d8d5eb6ec221be55,9978871a175e1a1f
Normal-2501.png,Normal,Normal/images/Normal-2501.png,Normal/masks/Normal-2501.png,True,G11277,train,b85685a1b9d0c60626a06e3ea3ffbfad,9968461119ef79b9
Normal-2502.png,Normal,Normal/images/Normal-2502.png,Normal/masks/Normal-2502.png,True,G11278,val,2342e058b884875b72f8c7e6f9a54ec6,d43d82c30f1d6d4b
Normal-2503.png,Normal,Normal/images/Normal-2503.png,Normal/masks/Normal-2503.png,True,G11279,val,7d097ed47a0dd22c4b028f6996a02e0c,802dd6035f867f1d
Normal-2504.png,Normal,Normal/images/Normal-2504.png,Normal/masks/Normal-2504.png,True,G11280,train,431d1c29681fc1edbc46f66b34cb4578,817b841a1b175e3f
Normal-2505.png,Normal,Normal/images/Normal-2505.png,Normal/masks/Normal-2505.png,True,G11281,val,c3a4d96ca90626d507cfc2167ba9b18a,dd6212495e594f1e
Normal-2506.png,Normal,Normal/images/Normal-2506.png,Normal/masks/Normal-2506.png,True,G11282,train,3ac867dabb4eeacbea56b7c261a42274,d979561e5c5a590a
Normal-2507.png,Normal,Normal/images/Normal-2507.png,Normal/masks/Normal-2507.png,True,G11283,train,9e9849c27605ffd33f84d4e798370ca1,902b471e4f5d78c9
Normal-2508.png,Normal,Normal/images/Normal-2508.png,Normal/masks/Normal-2508.png,True,G11284,test,150d5497dbab5dc7f37571849fee3879,b879c5561a455f38
Normal-2509.png,Normal,Normal/images/Normal-2509.png,Normal/masks/Normal-2509.png,True,G11285,train,e176099059fd2c630786dc7746c294c5,93631d153a587b72
Normal-251.png,Normal,Normal/images/Normal-251.png,Normal/masks/Normal-251.png,True,G11286,test,cc9af14ecbfab24ef008058e59642adc,d16ec628adcd0ec6
Normal-2510.png,Normal,Normal/images/Normal-2510.png,Normal/masks/Normal-2510.png,True,G11287,train,7b7cdaf93fb52b19be91f33aaa9658be,d12ec64b1c6f493c
Normal-2511.png,Normal,Normal/images/Normal-2511.png,Normal/masks/Normal-2511.png,True,G11288,train,a276ed9440c1bf5ca43c303310133f07,817d87031b3d472f
Normal-2512.png,Normal,Normal/images/Normal-2512.png,Normal/masks/Normal-2512.png,True,G11289,train,cfb2630715b89def893b32d666565bfc,81689e5a5f595a59
Normal-2513.png,Normal,Normal/images/Normal-2513.png,Normal/masks/Normal-2513.png,True,G11290,train,5415e54b6dfd4936485114bb6574491c,9929871e15075d5f
Normal-2514.png,Normal,Normal/images/Normal-2514.png,Normal/masks/Normal-2514.png,True,G11291,train,09a514e339b9c319998aaf161312ea81,9168061e7a1f7a5b
Normal-2515.png,Normal,Normal/images/Normal-2515.png,Normal/masks/Normal-2515.png,True,G11292,train,37ef1f05fe37ad5d8a9efea600a4eb59,d07d42571b524f4d
Normal-2516.png,Normal,Normal/images/Normal-2516.png,Normal/masks/Normal-2516.png,True,G11293,train,cde5d71872da83f3a0b6aec91a0ea68f,d96f461a5d0928ed
Normal-2517.png,Normal,Normal/images/Normal-2517.png,Normal/masks/Normal-2517.png,True,G11294,train,c4f29a478b2b505ed04b1675f7fe0fe3,996f061a1f4d78c5
Normal-2518.png,Normal,Normal/images/Normal-2518.png,Normal/masks/Normal-2518.png,True,G11295,train,080e2786ce6d27626f52ffb5896256d6,987f475f1b052d18
Normal-2519.png,Normal,Normal/images/Normal-2519.png,Normal/masks/Normal-2519.png,True,G11296,train,ae8c997349c428c51678870cf235e1c5,d979c61a114f3e58
Normal-252.png,Normal,Normal/images/Normal-252.png,Normal/masks/Normal-252.png,True,G11297,train,dd79409b456d1f3486f8895497de8891,c12f474b594169f5
Normal-2520.png,Normal,Normal/images/Normal-2520.png,Normal/masks/Normal-2520.png,True,G11298,train,335e4ee6153f42b6cd0f4a3ffebe0968,817bd61a190f3a6d
Normal-2521.png,Normal,Normal/images/Normal-2521.png,Normal/masks/Normal-2521.png,True,G11299,train,14432ac06584c5607424d799d17ff0fd,d92cc61b1d4e1b2d
Normal-2522.png,Normal,Normal/images/Normal-2522.png,Normal/masks/Normal-2522.png,True,G11300,train,5607212dc1e30ea8b6b1bae5f3b7e870,953ad2521b1bb939
Normal-2523.png,Normal,Normal/images/Normal-2523.png,Normal/masks/Normal-2523.png,True,G11301,val,c18564bd957bf5ab3179f61d3c294f38,993d851a1f4f1e19
Normal-2524.png,Normal,Normal/images/Normal-2524.png,Normal/masks/Normal-2524.png,True,G11302,val,69ecc361c236819aa2e342339e89bc63,d979c7d61a184f0a
Normal-2525.png,Normal,Normal/images/Normal-2525.png,Normal/masks/Normal-2525.png,True,G11303,val,e99498d54c82d48437124e6f5d35294e,dc0ad30b4f4a0f4f
Normal-2526.png,Normal,Normal/images/Normal-2526.png,Normal/masks/Normal-2526.png,True,G11304,test,360a77b5e1b7f567752737993ccd31a7,d96b861a1c6f3907
Normal-2527.png,Normal,Normal/images/Normal-2527.png,Normal/masks/Normal-2527.png,True,G11305,train,6958710a56f589ba24f1c022983cc04d,8979069e1b1d5f19
Normal-2528.png,Normal,Normal/images/Normal-2528.png,Normal/masks/Normal-2528.png,True,G11306,val,457c9885e886446c94be7cdfdc6845e1,816807be1f1e1f39
Normal-2529.png,Normal,Normal/images/Normal-2529.png,Normal/masks/Normal-2529.png,True,G11307,test,630d05b23bd821d63c0aab343c455d01,d03fc6535b424b1b
Normal-253.png,Normal,Normal/images/Normal-253.png,Normal/masks/Normal-253.png,True,G11308,train,fe92f8a22e8add05121dfb1bed88e0fc,95fcaf3a890c1996
Normal-2530.png,Normal,Normal/images/Normal-2530.png,Normal/masks/Normal-2530.png,True,G11309,train,3d0aceab65e825834a96d88379acb309,81785f561a1e1d1f
Normal-2531.png,Normal,Normal/images/Normal-2531.png,Normal/masks/Normal-2531.png,True,G11310,train,375d12fc1138b2a5627812d0da79adca,896d871e198d1f39
Normal-2532.png,Normal,Normal/images/Normal-2532.png,Normal/masks/Normal-2532.png,True,G11311,train,f6d2d9c928c2675425bafa726fbed8d9,c87d820399436f5f
Normal-2533.png,Normal,Normal/images/Normal-2533.png,Normal/masks/Normal-2533.png,True,G11312,train,beff733814d5c7e1d6eaf122eee055e0,cc3a530f544f1c4f
Normal-2534.png,Normal,Normal/images/Normal-2534.png,Normal/masks/Normal-2534.png,True,G11313,val,6db90a1850e8449f01e1221dfedcdd39,d16906961f951ad7
Normal-2535.png,Normal,Normal/images/Normal-2535.png,Normal/masks/Normal-2535.png,True,G11314,train,52bd71cafe044b9d549acf3149b773fc,817b05861e1a5f5f
Normal-2536.png,Normal,Normal/images/Normal-2536.png,Normal/masks/Normal-2536.png,True,G11315,train,4fb029e7cd8fd761e4aff7ff3ff73520,987a535a5d4d1c1b
Normal-2537.png,Normal,Normal/images/Normal-2537.png,Normal/masks/Normal-2537.png,True,G11316,train,97b0765bea9e95ec4916ebaab079376c,886b879e1f8c5e19
Normal-2538.png,Normal,Normal/images/Normal-2538.png,Normal/masks/Normal-2538.png,True,G11317,train,d55fcab09691653bf865fbf632c7d1bc,997985961e494f1e
Normal-2539.png,Normal,Normal/images/Normal-2539.png,Normal/masks/Normal-2539.png,True,G11318,train,5b6124479db2d8787a1168ba583c3c7b,dd28461f5a6d78c4
Normal-254.png,Normal,Normal/images/Normal-254.png,Normal/masks/Normal-254.png,True,G11319,train,81be9dbad51648a6ab489cb8c84e6f5e,9174c73acb3d7884
Normal-2540.png,Normal,Normal/images/Normal-2540.png,Normal/masks/Normal-2540.png,True,G11320,train,ea54165bdf283445735ca5a0abf7d1f4,996d87961a0f1e5a
Normal-2541.png,Normal,Normal/images/Normal-2541.png,Normal/masks/Normal-2541.png,True,G11321,val,9f0959c87a9dbcf12af59d511e8b1d18,9964421b451b7d6f
Normal-2542.png,Normal,Normal/images/Normal-2542.png,Normal/masks/Normal-2542.png,True,G11322,train,760131ff44bcd2df28445ccbf9487683,997d561a5f0e380d
Normal-2543.png,Normal,Normal/images/Normal-2543.png,Normal/masks/Normal-2543.png,True,G11323,test,ac3f8671209a62f342aabeb8b7353f62,9d2cc61b0d4c2ebd
Normal-2544.png,Normal,Normal/images/Normal-2544.png,Normal/masks/Normal-2544.png,True,G11324,test,390683c05e4d930f8dc61eaebb49835f,dd2cc25bc40b1d4f
Normal-2545.png,Normal,Normal/images/Normal-2545.png,Normal/masks/Normal-2545.png,True,G11325,test,56d1cfbf88078d522459fe9092ac9743,987d065a1a45ef78
Normal-2546.png,Normal,Normal/images/Normal-2546.png,Normal/masks/Normal-2546.png,True,G11326,train,42f7a390dcc3f645521b6410a920a086,c03d8b55bf4cb942
Normal-2547.png,Normal,Normal/images/Normal-2547.png,Normal/masks/Normal-2547.png,True,G11327,val,3eff04c8a4b0873d2f20cd082730e81f,903d474a396de1bc
Normal-2548.png,Normal,Normal/images/Normal-2548.png,Normal/masks/Normal-2548.png,True,G11328,train,7fecf814bb143cfe52e4b1777f047069,d939c6171f045b1b
Normal-2549.png,Normal,Normal/images/Normal-2549.png,Normal/masks/Normal-2549.png,True,G11329,test,1a798a66964506fba10d86f6130272f1,dd2895571e451f1c
Normal-255.png,Normal,Normal/images/Normal-255.png,Normal/masks/Normal-255.png,True,G11330,test,6bb52c2c0f66bc3887755666a06e45b9,907cc71e9ccd1cc5
Normal-2550.png,Normal,Normal/images/Normal-2550.png,Normal/masks/Normal-2550.png,True,G11331,train,c27206cba5372db42dda7dce52279e3e,d97d471ac44e3869
Normal-2551.png,Normal,Normal/images/Normal-2551.png,Normal/masks/Normal-2551.png,True,G11332,train,40cdfff470915edd8bd957eceb565b71,d53f420a4a1b5e2f
Normal-2552.png,Normal,Normal/images/Normal-2552.png,Normal/masks/Normal-2552.png,True,G11333,train,328b7ca11c1c2c3042344483493b98d4,c87957c61e1d4b19
Normal-2553.png,Normal,Normal/images/Normal-2553.png,Normal/masks/Normal-2553.png,True,G11334,train,a9d9a5bb1be44fdf99832247d7c525e5,9169861a1b177b3b
Normal-2554.png,Normal,Normal/images/Normal-2554.png,Normal/masks/Normal-2554.png,True,G11335,train,8cf93739d5b08276055145cccbcbbb3f,881b5b1cb99d9696
Normal-2555.png,Normal,Normal/images/Normal-2555.png,Normal/masks/Normal-2555.png,True,G11336,train,5c6f536c03cc14e3114202a5c66731b1,dd2c460b1b0daf1d
Normal-2556.png,Normal,Normal/images/Normal-2556.png,Normal/masks/Normal-2556.png,True,G11337,train,62a2f557b16d30fa93e91db8148ca077,9979c51e19495f38
Normal-2557.png,Normal,Normal/images/Normal-2557.png,Normal/masks/Normal-2557.png,True,G11338,train,3da4c2630470f1016e3318ba0f69eca0,d87d871e184d1e3a
Normal-2558.png,Normal,Normal/images/Normal-2558.png,Normal/masks/Normal-2558.png,True,G11339,train,a0c7f4921b2004aae41d356061ba15c8,816b871e98c73ab5
Normal-2559.png,Normal,Normal/images/Normal-2559.png,Normal/masks/Normal-2559.png,True,G11340,train,b2956519971423de32f80e88c54d7796,887d87121f1f1a5d
Normal-256.png,Normal,Normal/images/Normal-256.png,Normal/masks/Normal-256.png,True,G11341,train,f49c5627caba07e2d754b686d2b68ca2,816ac728cdcf4bc3
Normal-2560.png,Normal,Normal/images/Normal-2560.png,Normal/masks/Normal-2560.png,True,G11342,train,4c3c9c2bb6dced21652c69c8ece12398,d43f03931f0d0e3e
Normal-2561.png,Normal,Normal/images/Normal-2561.png,Normal/masks/Normal-2561.png,True,G11343,train,cd3d7ef3c75921623d5a23e23d191dee,dc2c520f1b4d6d1e
Normal-2562.png,Normal,Normal/images/Normal-2562.png,Normal/masks/Normal-2562.png,True,G11344,train,6c804ad1d80091bf8e5cae80ad5efbe1,987d871e194f1c55
Normal-2563.png,Normal,Normal/images/Normal-2563.png,Normal/masks/Normal-2563.png,True,G11345,train,cb3ac0b4c300cfd0415fcc46c7fb8405,9128165b7a6ef94a
Normal-2564.png,Normal,Normal/images/Normal-2564.png,Normal/masks/Normal-2564.png,True,G11346,val,08ccfe50b86dc63f3b003c23f74c3d21,d92b861f545f3849
Normal-2565.png,Normal,Normal/images/Normal-2565.png,Normal/masks/Normal-2565.png,True,G11347,val,7a34a55774f0b312c43f098d2fcb8c4b,817d060a1f4f7a79
Normal-2566.png,Normal,Normal/images/Normal-2566.png,Normal/masks/Normal-2566.png,True,G11348,test,23d3bf69469b034749710de844c6e6f5,9969c65e5cdc5a18
Normal-2567.png,Normal,Normal/images/Normal-2567.png,Normal/masks/Normal-2567.png,True,G11349,train,a3d1080cc8124b6b415319fe977f0d36,dc3dc30f4e4b0e4a
Normal-2568.png,Normal,Normal/images/Normal-2568.png,Normal/masks/Normal-2568.png,True,G11350,train,6244c3ea445d3d5c3f6bedb2e8765aed,993d461e184f197b
Normal-2569.png,Normal,Normal/images/Normal-2569.png,Normal/masks/Normal-2569.png,True,G11351,train,97ea9599c700a3e7f364a2234c86514c,d42ec35f0e0b1d4e
Normal-257.png,Normal,Normal/images/Normal-257.png,Normal/masks/Normal-257.png,True,G11352,train,deab3b0ef9693ae7724821f76947c11c,996dc71a8f692c4c
Normal-2570.png,Normal,Normal/images/Normal-2570.png,Normal/masks/Normal-2570.png,True,G11353,train,c96ab8e06c5027b56e40eb4a6018c4af,d43d43555f4a6c1a
Normal-2571.png,Normal,Normal/images/Normal-2571.png,Normal/masks/Normal-2571.png,True,G11354,train,4a23d4183064a52bf5c8fd0201226b93,913f07531b465e1b
Normal-2572.png,Normal,Normal/images/Normal-2572.png,Normal/masks/Normal-2572.png,True,G11355,train,32cba44d33a67fd5a654d136ed6c1e95,997907161ec75e98
Normal-2573.png,Normal,Normal/images/Normal-2573.png,Normal/masks/Normal-2573.png,True,G11356,train,718bbd01552f129556bc544fd2afabe4,d93dc21f1e4e1919
Normal-2574.png,Normal,Normal/images/Normal-2574.png,Normal/masks/Normal-2574.png,True,G11357,train,a20a2a1c8cc569f40bd3af677280ec7c,ddaf421f4613164b
Normal-2575.png,Normal,Normal/images/Normal-2575.png,Normal/masks/Normal-2575.png,True,G11358,train,9a3cf3dd8d9b72a11b8a9861393e30a3,fd5dc35e4c091926
Normal-2576.png,Normal,Normal/images/Normal-2576.png,Normal/masks/Normal-2576.png,True,G11359,train,4dcad970b0f619e55f9fdbc8cb1302d6,d969865e545a594e
Normal-2577.png,Normal,Normal/images/Normal-2577.png,Normal/masks/Normal-2577.png,True,G11360,val,ee8cb730591578dac1efb9b1e89422b1,c6abc9050b0f6f0f
Normal-2578.png,Normal,Normal/images/Normal-2578.png,Normal/masks/Normal-2578.png,True,G11361,val,d4e8ca4520ab22e8842b32b708b10be3,99ee065b4d584d78
Normal-2579.png,Normal,Normal/images/Normal-2579.png,Normal/masks/Normal-2579.png,True,G11362,val,0093d3279c5cb84a55eb88ac796c36c0,817b17961a977a98
Normal-258.png,Normal,Normal/images/Normal-258.png,Normal/masks/Normal-258.png,True,G11363,test,87a743e6208e8bb572d26b40ebbd8f49,957eaf1acd8d0883
Normal-2580.png,Normal,Normal/images/Normal-2580.png,Normal/masks/Normal-2580.png,True,G11364,test,7fd33b4bdf7f2b0d4a6cb6e32bb78eec,813bd786185d5f19
Normal-2581.png,Normal,Normal/images/Normal-2581.png,Normal/masks/Normal-2581.png,True,G11365,test,4a03ad276bbdf6c50bb9b28d74276132,d13d06041b5f6f39
Normal-2582.png,Normal,Normal/images/Normal-2582.png,Normal/masks/Normal-2582.png,True,G11366,train,c492eec8c9264b88dd1e2873dc1d7d1f,d93dc21b1d4f1919
Normal-2583.png,Normal,Normal/images/Normal-2583.png,Normal/masks/Normal-2583.png,True,G11367,train,bda9aa1c2fa58ededdff3f9927827bec,c979961e0e5a1c4f
Normal-2584.png,Normal,Normal/images/Normal-2584.png,Normal/masks/Normal-2584.png,True,G11368,train,97bc28684f6ab83760f8ca3dc5f43d75,91698618195f7b3b
Normal-2585.png,Normal,Normal/images/Normal-2585.png,Normal/masks/Normal-2585.png,True,G11369,train,cf42b6771784570a097b7ccc038d1870,d96945d75e0a4d0b
Normal-2586.png,Normal,Normal/images/Normal-2586.png,Normal/masks/Normal-2586.png,True,G11370,train,3e27ea1329d18d147f549d953bebef42,89fbc41ed5c094c7
Normal-2587.png,Normal,Normal/images/Normal-2587.png,Normal/masks/Normal-2587.png,True,G11371,test,7da5a91f09bbe7b6db7af50ae6e26654,887f875e48571e19
Normal-2588.png,Normal,Normal/images/Normal-2588.png,Normal/masks/Normal-2588.png,True,G11372,train,a7307f8395dc85ac5f78fa2bef2cb4d2,d97d471a124f3a98
Normal-2589.png,Normal,Normal/images/Normal-2589.png,Normal/masks/Normal-2589.png,True,G11373,train,57c8a389d40d193cdfff35c53b7185d1,d92984961e1f1d1f
Normal-259.png,Normal,Normal/images/Normal-259.png,Normal/masks/Normal-259.png,True,G11374,train,62ced65f532a9a2e8ea23ef3428b872d,d07aa73dcda440c7
Normal-2590.png,Normal,Normal/images/Normal-2590.png,Normal/masks/Normal-2590.png,True,G11375,train,6b3022ae3b88dcdb8285cdedf966c207,dc3dc20d0c6b3df0
Normal-2591.png,Normal,Normal/images/Normal-2591.png,Normal/masks/Normal-2591.png,True,G11376,test,d453be1eaffd3ef8cf66e69a5236032f,d82b570f1c28a9bd
Normal-2592.png,Normal,Normal/images/Normal-2592.png,Normal/masks/Normal-2592.png,True,G11377,train,033a424cbe5babc04ce3eefbd7603dc8,dd2ec25f1c0a1d3c
Normal-2593.png,Normal,Normal/images/Normal-2593.png,Normal/masks/Normal-2593.png,True,G11378,train,0183ad8355837647bd71b2624bca2dc4,dd2c420b1b6d8e1f
Normal-2594.png,Normal,Normal/images/Normal-2594.png,Normal/masks/Normal-2594.png,True,G11379,train,bb03beebfd497b0bb6a0d0c0ddda8a2b,9979861e164f5e1c
Normal-2595.png,Normal,Normal/images/Normal-2595.png,Normal/masks/Normal-2595.png,True,G11380,train,b36f99165cdec534864291a6a5199772,cd2ec00b1d6c1eed
Normal-2596.png,Normal,Normal/images/Normal-2596.png,Normal/masks/Normal-2596.png,True,G11381,train,bab213edd56c49b27593d54c4fbae563,997b85161a195b3b
Normal-2597.png,Normal,Normal/images/Normal-2597.png,Normal/masks/Normal-2597.png,True,G11382,train,72fba9eea723868acbad20bf3838c471,dd2c57c3430f4c4e
Normal-2598.png,Normal,Normal/images/Normal-2598.png,Normal/masks/Normal-2598.png,True,G11383,val,284cb080f1e63b49dc13d7181edaeed9,c03d3fc334f1b704
Normal-2599.png,Normal,Normal/images/Normal-2599.png,Normal/masks/Normal-2599.png,True,G11384,val,f3a326b0682e341408c3c1d7459f88c3,887907161e5d5e7a
Normal-26.png,Normal,Normal/images/Normal-26.png,Normal/masks/Normal-26.png,True,G11385,val,75ef4358fd95c551e51c4c93682bd25d,d93b860a196d5b39
Normal-260.png,Normal,Normal/images/Normal-260.png,Normal/masks/Normal-260.png,True,G11386,test,1a9ead4b60f26cfd9cb5a64aa3f1837b,8129ea39786939f9
Normal-2600.png,Normal,Normal/images/Normal-2600.png,Normal/masks/Normal-2600.png,True,G11387,train,72bc38c64675e190c87f7cce828ad832,9b681d105a797bd8
Normal-2601.png,Normal,Normal/images/Normal-2601.png,Normal/masks/Normal-2601.png,True,G11388,train,ebde0fea9315adc1921f10b1ac8022d7,8c4c430f4e4f597d
Normal-2602.png,Normal,Normal/images/Normal-2602.png,Normal/masks/Normal-2602.png,True,G11389,train,c9cedd145e6a7dcb46536a4497d951f4,996e861a1b475b1d
Normal-2603.png,Normal,Normal/images/Normal-2603.png,Normal/masks/Normal-2603.png,True,G11390,train,61e16f57efdd05269e290e44e67ed814,994d861a48cdbae7
Normal-2604.png,Normal,Normal/images/Normal-2604.png,Normal/masks/Normal-2604.png,True,G11391,train,ac69713fa297123fb534c73e6eef6e7c,997bc71a455e1907
Normal-2605.png,Normal,Normal/images/Normal-2605.png,Normal/masks/Normal-2605.png,True,G11392,train,df6a2dec3775c56e9dc6731bcc3c44fa,d82ac21b954f3d95
Normal-2606.png,Normal,Normal/images/Normal-2606.png,Normal/masks/Normal-2606.png,True,G11393,train,4b2569bc0b5784e4d1a13585049d96a9,d1284e5e38cf38cb
Normal-2607.png,Normal,Normal/images/Normal-2607.png,Normal/masks/Normal-2607.png,True,G11394,train,4ca2e440e41259e16bd40c54912a2296,897f061e582969f9
Normal-2608.png,Normal,Normal/images/Normal-2608.png,Normal/masks/Normal-2608.png,True,G11395,train,1b0c27882e59459abae74277e9a61c38,987dc61a455f186b
Normal-2609.png,Normal,Normal/images/Normal-2609.png,Normal/masks/Normal-2609.png,True,G11396,train,eef19f40ff56c87b74d4bc2711e133c8,d93f065a1c4d1b1d
Normal-261.png,Normal,Normal/images/Normal-261.png,Normal/masks/Normal-261.png,True,G11397,val,ec8c5d1ca9392699de0e178c26978142,d11fe4ca9cc8199e
Normal-2610.png,Normal,Normal/images/Normal-2610.png,Normal/masks/Normal-2610.png,True,G11398,train,e007d5607f18848eb4c170c400f77880,813b47c61b154e5f
Normal-2611.png,Normal,Normal/images/Normal-2611.png,Normal/masks/Normal-2611.png,True,G11399,train,9aad8543431170142aca71f56a56edde,d13f061e19ede1b0
Normal-2612.png,Normal,Normal/images/Normal-2612.png,Normal/masks/Normal-2612.png,True,G11400,val,28080a98b2f673bf3f43df4b6dfb53db,812d861e1f4e7a79
Normal-2613.png,Normal,Normal/images/Normal-2613.png,Normal/masks/Normal-2613.png,True,G11401,val,7f8fe471cdee0eaed9e07f014438f042,c03ccf4b180b6f3d
Normal-2614.png,Normal,Normal/images/Normal-2614.png,Normal/masks/Normal-2614.png,True,G11402,test,a093c0b752832d3020cd7dde3b4507ca,dc3c430b2cacfc3c
Normal-2615.png,Normal,Normal/images/Normal-2615.png,Normal/masks/Normal-2615.png,True,G11403,train,d780fb0d0c1d5e1b8b5b4960e0e85022,d979061e19ae38e9
Normal-2616.png,Normal,Normal/images/Normal-2616.png,Normal/masks/Normal-2616.png,True,G11404,test,2cd2aa2587f09bc677e0149cc64334cd,996d061a4d1b796b
Normal-2617.png,Normal,Normal/images/Normal-2617.png,Normal/masks/Normal-2617.png,True,G11405,test,0cacdd3374717c347d197bbcc8f808a9,db7857871e150e1c
Normal-2618.png,Normal,Normal/images/Normal-2618.png,Normal/masks/Normal-2618.png,True,G11406,train,e8d5908d0c3aac862921d2221ee46995,9938f7079a168a5e
Normal-2619.png,Normal,Normal/images/Normal-2619.png,Normal/masks/Normal-2619.png,True,G11407,train,c1e9b650eef3d87ebce337c9ee886ef3,80284717155f7fd9
Normal-262.png,Normal,Normal/images/Normal-262.png,Normal/masks/Normal-262.png,True,G11408,train,468233991db2fb95fe20a346bb256c40,d068c679c43c7cce
Normal-2620.png,Normal,Normal/images/Normal-2620.png,Normal/masks/Normal-2620.png,True,G11409,train,e0f3ae833f45581bd7ff9db2f783aac0,8029c7171f175f5a
Normal-2621.png,Normal,Normal/images/Normal-2621.png,Normal/masks/Normal-2621.png,True,G11410,train,f625bc5a81811d676b1019c57c00045f,d93cc7521b4e1e19
Normal-2622.png,Normal,Normal/images/Normal-2622.png,Normal/masks/Normal-2622.png,True,G11411,train,fb9cdea3f1bd963fc9f719ad25cc4ad7,996ec61b4d1f5845
Normal-2623.png,Normal,Normal/images/Normal-2623.png,Normal/masks/Normal-2623.png,True,G11412,train,0bac87c57f32f1df5a9300715a8cea54,916886575b5e7919
Normal-2624.png,Normal,Normal/images/Normal-2624.png,Normal/masks/Normal-2624.png,True,G11413,train,326c269edd2aaf19d52fa50fc378cab7,dd22431b4e0baa6f
Normal-2625.png,Normal,Normal/images/Normal-2625.png,Normal/masks/Normal-2625.png,True,G11414,test,1b0a01e0f19789745e96aa748500ef67,817f471a1c4f1f1a
Normal-2626.png,Normal,Normal/images/Normal-2626.png,Normal/masks/Normal-2626.png,True,G11415,train,6d44696b3fb1b7fe15e180060d5dbf20,d53dc20f1a4f1b1a
Normal-2627.png,Normal,Normal/images/Normal-2627.png,Normal/masks/Normal-2627.png,True,G11416,test,01de51f0e65af31edb03151a00fa28c9,817d07071e1b5f0f
Normal-2628.png,Normal,Normal/images/Normal-2628.png,Normal/masks/Normal-2628.png,True,G11417,train,7f629afefa5a1d7a80f0663fbd485b81,cc3d424b0fc26e3d
Normal-2629.png,Normal,Normal/images/Normal-2629.png,Normal/masks/Normal-2629.png,True,G11418,train,320ccdfd52b44f99d16d1027ad88ab62,c07f430a1d49bb5d
Normal-263.png,Normal,Normal/images/Normal-263.png,Normal/masks/Normal-263.png,True,G11419,train,8dd3a4da466ee99b52c97e5f5e486933,907cc61b8d6d3ccc
Normal-2630.png,Normal,Normal/images/Normal-2630.png,Normal/masks/Normal-2630.png,True,G11420,train,8724444bb85d07c26fd22e0d65adb3c5,d93b45161b596978
Normal-2631.png,Normal,Normal/images/Normal-2631.png,Normal/masks/Normal-2631.png,True,G11421,train,9bb135673f1419a728cec0145b0663d0,c93d461b164f1c4f
Normal-2632.png,Normal,Normal/images/Normal-2632.png,Normal/masks/Normal-2632.png,True,G11422,val,d3cb9f9758d76410c976b66c9da8618e,dc3dd2d14bd5490a
Normal-2633.png,Normal,Normal/images/Normal-2633.png,Normal/masks/Normal-2633.png,True,G11423,val,56914e202a6986f6db3414cb9a4735b6,c1ef075ad54a1547
Normal-2634.png,Normal,Normal/images/Normal-2634.png,Normal/masks/Normal-2634.png,True,G11424,val,6b15bb39f547089a506104f9819c46e2,997d071e184f1a79
Normal-2635.png,Normal,Normal/images/Normal-2635.png,Normal/masks/Normal-2635.png,True,G11425,test,22d9977e3502d085aedcecf3fa01d3a5,cd2ec249d04f1d4f
Normal-2636.png,Normal,Normal/images/Normal-2636.png,Normal/masks/Normal-2636.png,True,G11426,train,3cf03c3d7af4167be51a4544717f91ae,d42a574b4d4e4a6e
Normal-2637.png,Normal,Normal/images/Normal-2637.png,Normal/masks/Normal-2637.png,True,G11427,train,834fd0e29719577db6efd4505a157a1b,dd2b520f1c6839b9
Normal-2638.png,Normal,Normal/images/Normal-2638.png,Normal/masks/Normal-2638.png,True,G11428,train,a90aa0da961c6809e39a633f342fcf0e,9b69141a955a3bec
Normal-2639.png,Normal,Normal/images/Normal-2639.png,Normal/masks/Normal-2639.png,True,G11429,train,257f8a12835e8d80ad74d698448471df,995d065a1d5e5b0b
Normal-264.png,Normal,Normal/images/Normal-264.png,Normal/masks/Normal-264.png,True,G11430,train,aa852e8bb2b816c307b5ad39e38f11f9,816fcf5a89cb11c3
Normal-2640.png,Normal,Normal/images/Normal-2640.png,Normal/masks/Normal-2640.png,True,G11431,train,45e2f8e7c5c313c99a1445925cb67c93,d97dd21a444b1d4e
Normal-2641.png,Normal,Normal/images/Normal-2641.png,Normal/masks/Normal-2641.png,True,G11432,train,58ff87c638726423f5decb764895846a,d83f421f5c293c69
Normal-2642.png,Normal,Normal/images/Normal-2642.png,Normal/masks/Normal-2642.png,True,G11433,train,9197c96bb9f694dfe112ffe84e4fd927,d43c834f1d1b4c4f
Normal-2643.png,Normal,Normal/images/Normal-2643.png,Normal/masks/Normal-2643.png,True,G11434,train,504c9da31dfc7539211ab15a9c7b776f,887dc7190d0d3a5f
Normal-2644.png,Normal,Normal/images/Normal-2644.png,Normal/masks/Normal-2644.png,True,G11435,train,b6677a79395c0f4783053f71cec20b05,91695a561b1bb9c9
Normal-2645.png,Normal,Normal/images/Normal-2645.png,Normal/masks/Normal-2645.png,True,G11436,test,9a5863f3f536bee253ba27eb7be1b4da,d87d051e1ac97fa0
Normal-2646.png,Normal,Normal/images/Normal-2646.png,Normal/masks/Normal-2646.png,True,G11437,train,e30088c04cb8cd116d411ce5df8999b4,fd2e471f061d544c
Normal-2647.png,Normal,Normal/images/Normal-2647.png,Normal/masks/Normal-2647.png,True,G11438,train,f031fc7245e45a2e1a6d6f47887db554,c03d6bce3ee036c2
Normal-2648.png,Normal,Normal/images/Normal-2648.png,Normal/masks/Normal-2648.png,True,G11439,train,12f393c9f2b2b800a266b0bff67f0403,d128565b4b2d394f
Normal-2649.png,Normal,Normal/images/Normal-2649.png,Normal/masks/Normal-2649.png,True,G11440,train,6f3e1aa48f7cc7055a4c259c146c87cb,9168961b1d4f3979
Normal-265.png,Normal,Normal/images/Normal-265.png,Normal/masks/Normal-265.png,True,G11441,train,4bdc14c1d7632ef1376cd2e31862ac3c,903bc65a1ed8599d
Normal-2650.png,Normal,Normal/images/Normal-2650.png,Normal/masks/Normal-2650.png,True,G11442,val,36391b96a01e3bb31d7aec6ce9b83ca6,dd2c431b01471f5f
Normal-2651.png,Normal,Normal/images/Normal-2651.png,Normal/masks/Normal-2651.png,True,G11443,val,e76a6c8e484fd639aada4532300f065c,d0bd47181f0b1b4f
Normal-2652.png,Normal,Normal/images/Normal-2652.png,Normal/masks/Normal-2652.png,True,G11444,test,716936a501f712a802d8991698595a51,9968961a1f4e783b
Normal-2653.png,Normal,Normal/images/Normal-2653.png,Normal/masks/Normal-2653.png,True,G11445,val,0deb9eb19becafe80ee81f9e4a1f38aa,997dc61a95461f46
Normal-2654.png,Normal,Normal/images/Normal-2654.png,Normal/masks/Normal-2654.png,True,G11446,test,ecedd8faa0b54f5c48f0a21bbf75a6be,d979c61697061796
Normal-2655.png,Normal,Normal/images/Normal-2655.png,Normal/masks/Normal-2655.png,True,G11447,train,89c03b2eede7a871ad8dc8825a9d5b1c,8c2c530b454f5f4f
Normal-2656.png,Normal,Normal/images/Normal-2656.png,Normal/masks/Normal-2656.png,True,G11448,train,e719bf38ed6dd8eef6582c8c7d6bdc1e,dd2cc21b451f782d
Normal-2657.png,Normal,Normal/images/Normal-2657.png,Normal/masks/Normal-2657.png,True,G11449,train,11e6fcc66f2e2c645a8b89689f31f5ca,987b051e1e197bb1
Normal-2658.png,Normal,Normal/images/Normal-2658.png,Normal/masks/Normal-2658.png,True,G11450,train,9cf58018f7b295101f31cb9af7153cbb,dd2e43571e425f18
Normal-2659.png,Normal,Normal/images/Normal-2659.png,Normal/masks/Normal-2659.png,True,G11451,train,c45769f99d9aa50f782496ab4312ce3d,dd2e421b802fb9e6
Normal-266.png,Normal,Normal/images/Normal-266.png,Normal/masks/Normal-266.png,True,G11452,test,1d592ea1222d96fc68559bac06c70c6c,d07e873f2d2d2c84
Normal-2660.png,Normal,Normal/images/Normal-2660.png,Normal/masks/Normal-2660.png,True,G11453,train,dac91ab7f42c61b4a174c08042b0487e,d969061f1f0f194b
Normal-2661.png,Normal,Normal/images/Normal-2661.png,Normal/masks/Normal-2661.png,True,G11454,val,418bd178bdbf283166d8e9064f1bf8b7,8929021f5b5e7979
Normal-2662.png,Normal,Normal/images/Normal-2662.png,Normal/masks/Normal-2662.png,True,G11455,train,6334449313e8c2bc7f028df88271255b,817d461e5a495f39
Normal-2663.png,Normal,Normal/images/Normal-2663.png,Normal/masks/Normal-2663.png,True,G11456,train,b24d3b7628c1abc3c1df266997aa51f5,9939869e98abe8b8
Normal-2664.png,Normal,Normal/images/Normal-2664.png,Normal/masks/Normal-2664.png,True,G11457,val,e3a2397cfa8fd75c9165e7924f02702e,cd3dc21f460f1c4d
Normal-2665.png,Normal,Normal/images/Normal-2665.png,Normal/masks/Normal-2665.png,True,G11458,train,84dc05fd0925a2792df6e57bd1cdd2db,dd38d71b1d0a1f88
Normal-2666.png,Normal,Normal/images/Normal-2666.png,Normal/masks/Normal-2666.png,True,G11459,train,2dca005936b6b3cfb91f877df6348a2a,dd2a524b0e4a5b4f
Normal-2667.png,Normal,Normal/images/Normal-2667.png,Normal/masks/Normal-2667.png,True,G11460,train,02efa76be0c30d122546c25eac937554,d83bd61e581f5a18
Normal-2668.png,Normal,Normal/images/Normal-2668.png,Normal/masks/Normal-2668.png,True,G11461,train,6a7c50ab90408587e3e29d1be565b7a0,cc7d430b0e2e2d3d
Normal-2669.png,Normal,Normal/images/Normal-2669.png,Normal/masks/Normal-2669.png,True,G11462,train,535b6c6c1a3c337b288ca8768a4b3f59,dd2bd31a0a0b1b2f
Normal-267.png,Normal,Normal/images/Normal-267.png,Normal/masks/Normal-267.png,True,G11463,val,db579cce282f682bf5c3acff5180e60d,9129c61afa6b3969
Normal-2670.png,Normal,Normal/images/Normal-2670.png,Normal/masks/Normal-2670.png,True,G11464,test,b4dfb366c45f1b674927e386f2145d3d,dd2ec30b0d0c39bd
Normal-2671.png,Normal,Normal/images/Normal-2671.png,Normal/masks/Normal-2671.png,True,G11465,test,4d70d8c10ccc2664b0843742c1db1757,816806965a977adf
Normal-2672.png,Normal,Normal/images/Normal-2672.png,Normal/masks/Normal-2672.png,True,G11466,test,0f6f3ffa06816702987d4609066f4603,8978d65b180e3bb9
Normal-2673.png,Normal,Normal/images/Normal-2673.png,Normal/masks/Normal-2673.png,True,G11467,test,e6e71b1bc130ecafbcc73db87d9171c7,987b05161f4d5f19
Normal-2674.png,Normal,Normal/images/Normal-2674.png,Normal/masks/Normal-2674.png,True,G11468,train,a52cc9325b71923f66956be6d7d9f38d,d97dc64b1d4a1b18
Normal-2675.png,Normal,Normal/images/Normal-2675.png,Normal/masks/Normal-2675.png,True,G11469,train,502dc5c119576ed8d1a1967e8592b76e,dc2cd74a0c6ea87c
Normal-2676.png,Normal,Normal/images/Normal-2676.png,Normal/masks/Normal-2676.png,True,G11470,train,2e81afa488d2be6418963629d37f94f5,d969461a0d5e3766
Normal-2677.png,Normal,Normal/images/Normal-2677.png,Normal/masks/Normal-2677.png,True,G11471,train,692a1323f5ee436a9d25361246c57d3b,d92ad71e1c293939
Normal-2678.png,Normal,Normal/images/Normal-2678.png,Normal/masks/Normal-2678.png,True,G11472,train,f7ee4b9b6fcfc7a2be578f6a9c39a593,ad3cd34bc51a5d0a
Normal-2679.png,Normal,Normal/images/Normal-2679.png,Normal/masks/Normal-2679.png,True,G11473,train,478bbf940bb0bca561f56f05c55c7906,9928861e1f1e5e5d
Normal-268.png,Normal,Normal/images/Normal-268.png,Normal/masks/Normal-268.png,True,G11474,train,5a2f09c06cb7ab448e660483adae8570,d03dc33bb9e83892
Normal-2680.png,Normal,Normal/images/Normal-2680.png,Normal/masks/Normal-2680.png,True,G11475,train,61858ee1f52a04a3cdc68d676132dcef,9979861615cf3a95
Normal-2681.png,Normal,Normal/images/Normal-2681.png,Normal/masks/Normal-2681.png,True,G11476,train,9c672ec163ebc0c81d5fd69610abbd70,817f861e5a4f7862
Normal-2682.png,Normal,Normal/images/Normal-2682.png,Normal/masks/Normal-2682.png,True,G11477,train,6dd61457d07699a3529b8cea97e0f362,8c4c434b4f4f0f4f
Normal-2683.png,Normal,Normal/images/Normal-2683.png,Normal/masks/Normal-2683.png,True,G11478,train,8d9aec44f9977172c829e55babd2cc10,c939421f123db8eb
Normal-2684.png,Normal,Normal/images/Normal-2684.png,Normal/masks/Normal-2684.png,True,G11479,train,b98223c6cfafeb684cdad95842b7109f,817a5616185f793b
Normal-2685.png,Normal,Normal/images/Normal-2685.png,Normal/masks/Normal-2685.png,True,G11480,train,361d9710209541a7690f3eeeb9f6088d,802f06584b6bf9e7
Normal-2686.png,Normal,Normal/images/Normal-2686.png,Normal/masks/Normal-2686.png,True,G11481,train,f416e21d8ef7720382af016edf2fac0a,9979871e1c0d1ab7
Normal-2687.png,Normal,Normal/images/Normal-2687.png,Normal/masks/Normal-2687.png,True,G11482,val,aee6f917a724f9c8bef36dda7d0d0c9b,d87fd64358525b18
Normal-2688.png,Normal,Normal/images/Normal-2688.png,Normal/masks/Normal-2688.png,True,G11483,val,54384845d9682c89aa96c1df037a19d4,9928871a0f3b1f4f
Normal-2689.png,Normal,Normal/images/Normal-2689.png,Normal/masks/Normal-2689.png,True,G11484,test,4f11c010e7ef6af0a32f06bf0a3f6a81,9139a7c61e1ec54b
Normal-269.png,Normal,Normal/images/Normal-269.png,Normal/masks/Normal-269.png,True,G11485,val,a68a55431fc561e3c8faaa370a7c4c98,916bc67a39696968
Normal-2690.png,Normal,Normal/images/Normal-2690.png,Normal/masks/Normal-2690.png,True,G11486,train,60f9413d963c3390ddf0e10e9b52d5f4,f96f865a584948f8
Normal-2691.png,Normal,Normal/images/Normal-2691.png,Normal/masks/Normal-2691.png,True,G11487,train,f3127ef32f150eb2d90c43ea33d0173f,dd68431b1b053f1d
Normal-2692.png,Normal,Normal/images/Normal-2692.png,Normal/masks/Normal-2692.png,True,G11488,train,2e8119907cc5dfce8e566d9b5a666855,817b861a195d5f39
Normal-2693.png,Normal,Normal/images/Normal-2693.png,Normal/masks/Normal-2693.png,True,G11489,train,7e67f7ae5b9cc207a1c243035560dc3b,953f06031b0f5f1d
Normal-2694.png,Normal,Normal/images/Normal-2694.png,Normal/masks/Normal-2694.png,True,G11490,val,a052b44b31979f8a447a78c5b9b955f5,d17c46431b0f4e3d
Normal-2695.png,Normal,Normal/images/Normal-2695.png,Normal/masks/Normal-2695.png,True,G11491,train,e81bd6d1ac6e85d85fe7b4cdfc320c6e,dd2ed25b5b404b1c
Normal-2696.png,Normal,Normal/images/Normal-2696.png,Normal/masks/Normal-2696.png,True,G11492,train,2779109c139d17c61d453b6df40b2ae4,dd2c42821f4d6f1e
Normal-2697.png,Normal,Normal/images/Normal-2697.png,Normal/masks/Normal-2697.png,True,G11493,train,73de0363f7549a4543128d3884247023,957c075a095fcd38
Normal-2698.png,Normal,Normal/images/Normal-2698.png,Normal/masks/Normal-2698.png,True,G11494,test,8e8556537465235e8a9b7a37bfbb05ba,817f061e5f687839
Normal-2699.png,Normal,Normal/images/Normal-2699.png,Normal/masks/Normal-2699.png,True,G11495,train,d0f1103505549aba58efb71df58dc8d4,d979075e1a4d4939
Normal-27.png,Normal,Normal/images/Normal-27.png,Normal/masks/Normal-27.png,True,G11496,test,1346d143e212564c083edf7e16aa2c4e,bd00d31f541a5f1e
Normal-270.png,Normal,Normal/images/Normal-270.png,Normal/masks/Normal-270.png,True,G11497,train,4059bc1ec4ecf240656ca8aa83b92145,917b5e5e1909398b
Normal-2700.png,Normal,Normal/images/Normal-2700.png,Normal/masks/Normal-2700.png,True,G11498,train,32975fc8fec699effe7285ea042bfc4d,d92e06431b7d6a39
Normal-2701.png,Normal,Normal/images/Normal-2701.png,Normal/masks/Normal-2701.png,True,G11499,train,91d0e4bfdb015dcbae83bb40c5ad10dc,c868c2561f863f79
Normal-2702.png,Normal,Normal/images/Normal-2702.png,Normal/masks/Normal-2702.png,True,G11500,val,41fd8cbf3b609be6668761c0d20fa0ed,cc2cc30bed4c3ee4
Normal-2703.png,Normal,Normal/images/Normal-2703.png,Normal/masks/Normal-2703.png,True,G11501,train,44fe3d7662ad2fe1a2fc7c142d412c96,dc7d83521c470b1f
Normal-2704.png,Normal,Normal/images/Normal-2704.png,Normal/masks/Normal-2704.png,True,G11502,train,c53ae10d39548d899eb8a1f1f7cc489a,dd2cc71b455a5c4c
Normal-2705.png,Normal,Normal/images/Normal-2705.png,Normal/masks/Normal-2705.png,True,G11503,train,e350bc67e3463c04f4b89b3d527b6567,d83f420f1c2b3d3c
Normal-2706.png,Normal,Normal/images/Normal-2706.png,Normal/masks/Normal-2706.png,True,G11504,train,e9181fee552e24a01895367a32f6886d,987f87121e475a1b
Normal-2707.png,Normal,Normal/images/Normal-2707.png,Normal/masks/Normal-2707.png,True,G11505,train,aa8ccd7c8e6f53092bebd706ea80b0ef,d97b47161e0e4a39
Normal-2708.png,Normal,Normal/images/Normal-2708.png,Normal/masks/Normal-2708.png,True,G11506,train,273ff04c02821fa951c1429296bf14e8,d03d42c50b1f4d3f
Normal-2709.png,Normal,Normal/images/Normal-2709.png,Normal/masks/Normal-2709.png,True,G11507,val,af0f71d61a31e0b645ef680b3456c4fc,d9ad831a051bc71f
Normal-271.png,Normal,Normal/images/Normal-271.png,Normal/masks/Normal-271.png,True,G11508,test,d68ea33d59a63a4653f3ac4285dc5a4e,d465871bcd0d6cc5
Normal-2710.png,Normal,Normal/images/Normal-2710.png,Normal/masks/Normal-2710.png,True,G11509,train,f813671d432d3c819a23d14b249b6e89,d42d0ac33dd83cf4
Normal-2711.png,Normal,Normal/images/Normal-2711.png,Normal/masks/Normal-2711.png,True,G11510,train,4c85d38cbd2a4e8da94acd6ddf207090,817b44145a5b7bd9
Normal-2712.png,Normal,Normal/images/Normal-2712.png,Normal/masks/Normal-2712.png,True,G11511,train,cde41a0e2cc6ea5002430f8f648b3e38,dd2dd0131f474a1d
Normal-2713.png,Normal,Normal/images/Normal-2713.png,Normal/masks/Normal-2713.png,True,G11512,train,56b794d8082c71a7c7332cd443d5d920,d97b861e584b185b
Normal-2714.png,Normal,Normal/images/Normal-2714.png,Normal/masks/Normal-2714.png,True,G11513,val,8d77199da3ccacf118b78fde99097a37,997bc7161a491b1e
Normal-2715.png,Normal,Normal/images/Normal-2715.png,Normal/masks/Normal-2715.png,True,G11514,train,f20eaf8aaaa37f8dc8ea0f0cd5b60f6a,8139de16580f6b3b
Normal-2716.png,Normal,Normal/images/Normal-2716.png,Normal/masks/Normal-2716.png,True,G11515,test,0437f7f755b7db469a4fc8c17df01276,89ef06184d1c4f5f
Normal-2717.png,Normal,Normal/images/Normal-2717.png,Normal/masks/Normal-2717.png,True,G11516,train,4416541c0f6da6264c8b062c79f2f4b8,812bc6d61acefa1a
Normal-2718.png,Normal,Normal/images/Normal-2718.png,Normal/masks/Normal-2718.png,True,G11517,train,da2b2a2d8bfeff7539f39fd17bfb2ae1,c97ac61e180d3fb8
Normal-2719.png,Normal,Normal/images/Normal-2719.png,Normal/masks/Normal-2719.png,True,G11518,train,c3035b422ba765b1330e754b32d54382,887f851a170f1f1d
Normal-272.png,Normal,Normal/images/Normal-272.png,Normal/masks/Normal-272.png,True,G11519,train,b4a5415c0f09fa5911cdae96b36a2151,903ac75a3cc14bbd
Normal-2720.png,Normal,Normal/images/Normal-2720.png,Normal/masks/Normal-2720.png,True,G11520,train,5a938636fb4c00cef14ec73559d778a7,996d065e08ed7aa5
Normal-2721.png,Normal,Normal/images/Normal-2721.png,Normal/masks/Normal-2721.png,True,G11521,train,54e716021e58dca3cb3a18864c22e6a2,d979d217114f1a59
Normal-2722.png,Normal,Normal/images/Normal-2722.png,Normal/masks/Normal-2722.png,True,G11522,val,226e553916301b6bc7c0baae1cd2df46,d92986071f971e9c
Normal-2723.png,Normal,Normal/images/Normal-2723.png,Normal/masks/Normal-2723.png,True,G11523,val,a43503a155cf5a98360d43b835704dcd,982fc71ac51f1d43
Normal-2724.png,Normal,Normal/images/Normal-2724.png,Normal/masks/Normal-2724.png,True,G11524,train,689280db56f64dfd840d149ae5b7e9f4,dd2ec29b9a80aa9d
Normal-2725.png,Normal,Normal/images/Normal-2725.png,Normal/masks/Normal-2725.png,True,G11525,test,5b55856d5fad00f1965e1d724c1b3070,893ec61b194d1e79
Normal-2726.png,Normal,Normal/images/Normal-2726.png,Normal/masks/Normal-2726.png,True,G11526,test,43cabf1a208566bc3b612d354ce97f2a,9d3c461b4a6c78e5
Normal-2727.png,Normal,Normal/images/Normal-2727.png,Normal/masks/Normal-2727.png,True,G11527,train,85f53ab59cc6cd562bc9c341a5cf6975,8868475a1f4f7978
Normal-2728.png,Normal,Normal/images/Normal-2728.png,Normal/masks/Normal-2728.png,True,G11528,train,fcb987ac916a97b373a5bdeed00a72ef,c139ce07194ffe18
Normal-2729.png,Normal,Normal/images/Normal-2729.png,Normal/masks/Normal-2729.png,True,G11529,train,d22df3ff1ef05a9b2388953a9dcdc744,897d071a155e39e9
Normal-273.png,Normal,Normal/images/Normal-273.png,Normal/masks/Normal-273.png,True,G11530,train,d2cac81274aa0b745683a03ce922c77e,d42ae32fec693c60
Normal-2730.png,Normal,Normal/images/Normal-2730.png,Normal/masks/Normal-2730.png,True,G11531,train,16d5c5431d57449c532c6469254a0cee,8179c646da15fe46
Normal-2731.png,Normal,Normal/images/Normal-2731.png,Normal/masks/Normal-2731.png,True,G11532,train,869e025c0f9d5ea3d3d20a43e075a4b8,993dc61a15451bbd
Normal-2732.png,Normal,Normal/images/Normal-2732.png,Normal/masks/Normal-2732.png,True,G11533,train,3444da7902ef866cd3670553cc193b26,dc3d534b4f4a0c2e
Normal-2733.png,Normal,Normal/images/Normal-2733.png,Normal/masks/Normal-2733.png,True,G11534,train,c6dcc28507fd52d1ec5804efc47e1657,a83dc6531b454f1e
Normal-2734.png,Normal,Normal/images/Normal-2734.png,Normal/masks/Normal-2734.png,True,G11535,train,ea55d81af506fa0ddfa1be6e7656205f,987b87161c5f1b19
Normal-2735.png,Normal,Normal/images/Normal-2735.png,Normal/masks/Normal-2735.png,True,G11536,train,930721167bc80f23d1b9fed507b1215d,dd2c52d35b5d084d
Normal-2736.png,Normal,Normal/images/Normal-2736.png,Normal/masks/Normal-2736.png,True,G11537,train,b2a590352112b1f65b137e9ad07c5f33,987bc71e16071e1b
Normal-2737.png,Normal,Normal/images/Normal-2737.png,Normal/masks/Normal-2737.png,True,G11538,train,3e968092fcba15a42b90eb4544dbd79b,d12f860a1d0f595f
Normal-2738.png,Normal,Normal/images/Normal-2738.png,Normal/masks/Normal-2738.png,True,G11539,train,d668fd4a74b005559d752d949327fb77,9978069e1b0dea5b
Normal-2739.png,Normal,Normal/images/Normal-2739.png,Normal/masks/Normal-2739.png,True,G11540,train,19c422f6f1a32fc09d62f3010383a25c,cc3d420f2c6a3dbc
Normal-274.png,Normal,Normal/images/Normal-274.png,Normal/masks/Normal-274.png,True,G11541,val,71a18ff9116db9a0324c510834291694,917ac65a595d11d5
Normal-2740.png,Normal,Normal/images/Normal-2740.png,Normal/masks/Normal-2740.png,True,G11542,val,f7d865ade18dbccd34694a754308240b,9978065a5be9e8a5
Normal-2741.png,Normal,Normal/images/Normal-2741.png,Normal/masks/Normal-2741.png,True,G11543,val,8756fdf92aba51758ed714f21cf5adb6,91381e1f3a7ae1b2
Normal-2742.png,Normal,Normal/images/Normal-2742.png,Normal/masks/Normal-2742.png,True,G11544,test,443b4e555652d06fe81278da9baeef56,997bc5161a417f1a
Normal-2743.png,Normal,Normal/images/Normal-2743.png,Normal/masks/Normal-2743.png,True,G11545,test,b840f1ea1758a24b20f44f92ba3b3fd1,dc2e425d424f1e4f
Normal-2744.png,Normal,Normal/images/Normal-2744.png,Normal/masks/Normal-2744.png,True,G11546,test,c1145720056dd4567224a5838650e91e,916ac61c545e5a5f
Normal-2745.png,Normal,Normal/images/Normal-2745.png,Normal/masks/Normal-2745.png,True,G11547,train,d5e91ccb159cb35e097ddf0b699fa45c,dc2cc34b1b4b1f1a
Normal-2746.png,Normal,Normal/images/Normal-2746.png,Normal/masks/Normal-2746.png,True,G11548,train,a6d89580ff6ca21677db88d161253e4f,9c3cc31b1d471e4d
Normal-2747.png,Normal,Normal/images/Normal-2747.png,Normal/masks/Normal-2747.png,True,G11549,train,89b1cf65c7f68a0804458dfad69d9da2,dc7dd2025f0d0a1f
Normal-2748.png,Normal,Normal/images/Normal-2748.png,Normal/masks/Normal-2748.png,True,G11550,train,fd9b8e65a943e59c4a0808e8f67abb81,dc7c8719041e79ae
Normal-2749.png,Normal,Normal/images/Normal-2749.png,Normal/masks/Normal-2749.png,True,G11551,train,e2299e2ef11e21fd05f17c04ef7dddf5,d96987871ac54b93
Normal-275.png,Normal,Normal/images/Normal-275.png,Normal/masks/Normal-275.png,True,G11552,train,f2562b4280c2f6adea521c4991b0782f,d574873bad2d2c24
Normal-2750.png,Normal,Normal/images/Normal-2750.png,Normal/masks/Normal-2750.png,True,G11553,train,2f6229b20b71fcca0923d0c80cf953e3,9979861e5a0d5a79
Normal-2751.png,Normal,Normal/images/Normal-2751.png,Normal/masks/Normal-2751.png,True,G11554,train,67603be9c6dabd6bf4ee36a2fa93abf3,dd2c525f424b1d4e
Normal-2752.png,Normal,Normal/images/Normal-2752.png,Normal/masks/Normal-2752.png,True,G11555,train,5dc141102806c464bcd1adccf5ec1a03,d97945975e401f1a
Normal-2753.png,Normal,Normal/images/Normal-2753.png,Normal/masks/Normal-2753.png,True,G11556,train,e4f7a2b2a9049eeddf90783890238c4b,dd2cc6531b464b1d
Normal-2754.png,Normal,Normal/images/Normal-2754.png,Normal/masks/Normal-2754.png,True,G11557,train,0c8269a2e592c31f7d6b46a2107cb0fe,9939861c1f683aeb
Normal-2755.png,Normal,Normal/images/Normal-2755.png,Normal/masks/Normal-2755.png,True,G11558,train,026a94acb3f27423b720adaeeebc3f28,9178061a5b59797d
Normal-2756.png,Normal,Normal/images/Normal-2756.png,Normal/masks/Normal-2756.png,True,G11559,train,49d321022cd5fbf0e5f70b3df27430ea,fc3cd34b47434d0c
Normal-2757.png,Normal,Normal/images/Normal-2757.png,Normal/masks/Normal-2757.png,True,G11560,train,3aafff79e3d90a2f94398effafefabbe,c83d420b174f1b7d
Normal-2758.png,Normal,Normal/images/Normal-2758.png,Normal/masks/Normal-2758.png,True,G11561,val,641453c49a2992513577179f49fb4d00,d979c5461ac44b3d
Normal-2759.png,Normal,Normal/images/Normal-2759.png,Normal/masks/Normal-2759.png,True,G11562,test,ba9476d15876ae8fe9320d4b7928ad03,d929d6071e063b1f
Normal-276.png,Normal,Normal/images/Normal-276.png,Normal/masks/Normal-276.png,True,G11563,val,bcbe74b049fa656e5ebb27a39efbebb0,c1fece5ac45859c2
Normal-2760.png,Normal,Normal/images/Normal-2760.png,Normal/masks/Normal-2760.png,True,G11564,val,926d1ab90e0f9bfcd5e58c15860d9def,997b84021b8e5e3f
Normal-2761.png,Normal,Normal/images/Normal-2761.png,Normal/masks/Normal-2761.png,True,G11565,test,def6a7767f82bf56c8ca729e65c0544e,fd2d46c24a4e4a5b
Normal-2762.png,Normal,Normal/images/Normal-2762.png,Normal/masks/Normal-2762.png,True,G11566,test,918155eb891465101567a02cf481247d,9179c6161f464f1b
Normal-2763.png,Normal,Normal/images/Normal-2763.png,Normal/masks/Normal-2763.png,True,G11567,train,38f976d3a6f4093cbf14ca012c26f42e,997d46015b574d39
Normal-2764.png,Normal,Normal/images/Normal-2764.png,Normal/masks/Normal-2764.png,True,G11568,train,a0cf97074ee0b633ed1ce3050a19d48e,997bc5161c535b19
Normal-2765.png,Normal,Normal/images/Normal-2765.png,Normal/masks/Normal-2765.png,True,G11569,train,ba3056048a7d96af5924d45a9da06f60,c92ad45b554a1f0f
Normal-2766.png,Normal,Normal/images/Normal-2766.png,Normal/masks/Normal-2766.png,True,G11570,train,5f17d3599f39fe3bc1bff2ed4b9a812e,d92b061e5c5c78e9
Normal-2767.png,Normal,Normal/images/Normal-2767.png,Normal/masks/Normal-2767.png,True,G11571,train,936c0cd356a76add24cf0a3a60e73650,d93d521e184e4f39
Normal-2768.png,Normal,Normal/images/Normal-2768.png,Normal/masks/Normal-2768.png,True,G11572,train,d36f1c31729cd226fdc412e303a1fcc6,d93bc61a186c39bc
Normal-2769.png,Normal,Normal/images/Normal-2769.png,Normal/masks/Normal-2769.png,True,G11573,train,49d30bca01434323e4097427aedb7c22,d969471ad5475c45
Normal-277.png,Normal,Normal/images/Normal-277.png,Normal/masks/Normal-277.png,True,G11574,train,e138c201b371b4d24e30a0171c649e08,902ac25bdd693d6c
Normal-2770.png,Normal,Normal/images/Normal-2770.png,Normal/masks/Normal-2770.png,True,G11575,train,4a114583a1c7fdb7ec3d02cbc51d9cd5,d03d42531d3c6b3d
Normal-2771.png,Normal,Normal/images/Normal-2771.png,Normal/masks/Normal-2771.png,True,G11576,train,0ba7ac1927a6cdaf0ef6d0580091cdbb,dc2c430b0dcf2b3d
Normal-2772.png,Normal,Normal/images/Normal-2772.png,Normal/masks/Normal-2772.png,True,G11577,train,00db57281b54f752c8e4292bd3007776,c97d461b1c4b39c9
Normal-2773.png,Normal,Normal/images/Normal-2773.png,Normal/masks/Normal-2773.png,True,G11578,val,c74aad77fabc74af850b99d3c8e63b27,997c861a1b4d7a39
Normal-2774.png,Normal,Normal/images/Normal-2774.png,Normal/masks/Normal-2774.png,True,G11579,train,ca69c447d998d85a504d909fe3a7f8fa,81791e871a1fc75a
Normal-2775.png,Normal,Normal/images/Normal-2775.png,Normal/masks/Normal-2775.png,True,G11580,val,fdf17164ba9c9f86b4565a84dbc0bbde,cc3d4213464b1fcf
Normal-2776.png,Normal,Normal/images/Normal-2776.png,Normal/masks/Normal-2776.png,True,G11581,train,a28c2f6add0c6e6e01178d73cb78f640,d47cc2535b4b4b0b
Normal-2777.png,Normal,Normal/images/Normal-2777.png,Normal/masks/Normal-2777.png,True,G11582,train,396e5aac7d7d3b737c969e3b331544fb,a013405fd73a5dcb
Normal-2778.png,Normal,Normal/images/Normal-2778.png,Normal/masks/Normal-2778.png,True,G11583,train,e83982e065e34181cdc674222515e5b6,997b961e1c1a39a9
Normal-2779.png,Normal,Normal/images/Normal-2779.png,Normal/masks/Normal-2779.png,True,G11584,val,bff5db80a9bfd48b4579c0d06843d22b,8138564f3a3bec69
Normal-278.png,Normal,Normal/images/Normal-278.png,Normal/masks/Normal-278.png,True,G11585,test,62a66ea3e1d9d91210cf3e8f9da54d1b,d46c874fcd4c4cc6
Normal-2780.png,Normal,Normal/images/Normal-2780.png,Normal/masks/Normal-2780.png,True,G11586,test,a0e07033aa71ed4cc2f58c1a86466740,9968161bde79e168
Normal-2781.png,Normal,Normal/images/Normal-2781.png,Normal/masks/Normal-2781.png,True,G11587,test,795edf9e4c74008841b4eae3d9067952,9d3dc61b59452e38
Normal-2782.png,Normal,Normal/images/Normal-2782.png,Normal/masks/Normal-2782.png,True,G11588,val,24657e84522afb74dd10d7e528d43e4b,dc3d435d0b0e6d38
Normal-2783.png,Normal,Normal/images/Normal-2783.png,Normal/masks/Normal-2783.png,True,G11589,train,4d36138d638a424d02b004426dbd6fbe,cd2cd20b5d491d4f
Normal-2784.png,Normal,Normal/images/Normal-2784.png,Normal/masks/Normal-2784.png,True,G11590,test,66ba99f8d13c9f1f5f55ac7bd36a521b,992d07115f4e6c79
Normal-2785.png,Normal,Normal/images/Normal-2785.png,Normal/masks/Normal-2785.png,True,G11591,train,65bb991f8ab8f1b03294f61ca5d0ca93,d42dc2075da84afd
Normal-2786.png,Normal,Normal/images/Normal-2786.png,Normal/masks/Normal-2786.png,True,G11592,train,90f162e301cbf7cfeb427119a56bb5da,d93d061a1a4d5e3e
Normal-2787.png,Normal,Normal/images/Normal-2787.png,Normal/masks/Normal-2787.png,True,G11593,train,99ac05071d2443bae72cdd276d97fb8d,dc2cd34f4a0d1e2d
Normal-2788.png,Normal,Normal/images/Normal-2788.png,Normal/masks/Normal-2788.png,True,G11594,train,8069c8cab60087e959bd038eb2b93483,896e461a5f6c78e1
Normal-2789.png,Normal,Normal/images/Normal-2789.png,Normal/masks/Normal-2789.png,True,G11595,train,2769f61c5fe671f1724469524513e9ad,dd2cc21b054b1f4f
Normal-279.png,Normal,Normal/images/Normal-279.png,Normal/masks/Normal-279.png,True,G11596,train,5394fc512737486ae562b76a0695b6ee,916f8f1a0f4d1c4d
Normal-2790.png,Normal,Normal/images/Normal-2790.png,Normal/masks/Normal-2790.png,True,G11597,train,2d78cc266230af0e1c679158d9e8b940,882c475b1b4c7d3d
Normal-2791.png,Normal,Normal/images/Normal-2791.png,Normal/masks/Normal-2791.png,True,G11598,test,c5a8106743634c4c0af3c7963c460000,ccb5430a4d6e2e6b
Normal-2792.png,Normal,Normal/images/Normal-2792.png,Normal/masks/Normal-2792.png,True,G11599,train,134f8da8c9c06386bb0cbe2aa7251e8f,c169c21b4d4f3cdc
Normal-2793.png,Normal,Normal/images/Normal-2793.png,Normal/masks/Normal-2793.png,True,G11600,train,4c9ec793c6a8f702f17ece75ab62af78,d969161a5d5a194f
Normal-2794.png,Normal,Normal/images/Normal-2794.png,Normal/masks/Normal-2794.png,True,G11601,train,136418072350d2ef4f7ee9ffffa54224,817956461b5d0f1f
Normal-2795.png,Normal,Normal/images/Normal-2795.png,Normal/masks/Normal-2795.png,True,G11602,train,50159742060bcee628196dec94a02f6d,d93f521e4b190a3b
Normal-2796.png,Normal,Normal/images/Normal-2796.png,Normal/masks/Normal-2796.png,True,G11603,train,03855d5cd7fab190b56e4b8a2409293f,d97b461e1e4b1a19
Normal-2797.png,Normal,Normal/images/Normal-2797.png,Normal/masks/Normal-2797.png,True,G11604,train,ba8ef6b76a44d4f7a0a3560997fdf8fe,9968161ac75a575e
Normal-2798.png,Normal,Normal/images/Normal-2798.png,Normal/masks/Normal-2798.png,True,G11605,train,8d160a8b87769e27ce01cceb27333bac,d93bc61e86c49aa9
Normal-2799.png,Normal,Normal/images/Normal-2799.png,Normal/masks/Normal-2799.png,True,G11606,val,a60a55eed21fb89cccfb3381b4c80211,8879471e1b497bb8
Normal-28.png,Normal,Normal/images/Normal-28.png,Normal/masks/Normal-28.png,True,G11607,val,92de83da604b021c2688253be1a0339c,91e80617161f1f5f
Normal-280.png,Normal,Normal/images/Normal-280.png,Normal/masks/Normal-280.png,True,G11608,test,a54b148c6f4751ccbe9645b4477a9eb2,907fc61e1e491d5c
Normal-2800.png,Normal,Normal/images/Normal-2800.png,Normal/masks/Normal-2800.png,True,G11609,train,b84c5ec38993195dee9f3931d59cacc8,dc2d421f0bcc3f38
Normal-2801.png,Normal,Normal/images/Normal-2801.png,Normal/masks/Normal-2801.png,True,G11610,train,36c8c423fc43d4dc1c5e202e8f473675,997d861e184d5b39
Normal-2802.png,Normal,Normal/images/Normal-2802.png,Normal/masks/Normal-2802.png,True,G11611,train,0cc5881081789984d26ef375fd94de55,cc6c531b4d4e5d45
Normal-2803.png,Normal,Normal/images/Normal-2803.png,Normal/masks/Normal-2803.png,True,G11612,train,293d598d8b64f40e09daba9dc9f4b56d,886a47561f455f1b
Normal-2804.png,Normal,Normal/images/Normal-2804.png,Normal/masks/Normal-2804.png,True,G11613,train,dfda35effc9bf2b2a5829cffe9a26cfa,916c8e1efa4fe805
Normal-2805.png,Normal,Normal/images/Normal-2805.png,Normal/masks/Normal-2805.png,True,G11614,train,aa1608606f0ffae75d8cdff78166ebe3,902f46175a4f1b5d
Normal-2806.png,Normal,Normal/images/Normal-2806.png,Normal/masks/Normal-2806.png,True,G11615,train,3982f0c38ae91b24e2c9ea27a0b89b2f,d03dc35b1d4d0f1c
Normal-2807.png,Normal,Normal/images/Normal-2807.png,Normal/masks/Normal-2807.png,True,G11616,train,db111cd0026c70a1dd5acc5da13b917e,d965c71ac5471d45
Normal-2808.png,Normal,Normal/images/Normal-2808.png,Normal/masks/Normal-2808.png,True,G11617,train,9e6c6d6be23976431e48626fbf71bc75,d93d461b194f3878
Normal-2809.png,Normal,Normal/images/Normal-2809.png,Normal/masks/Normal-2809.png,True,G11618,train,4acd5f93f0a81ef4da293cfa1149e233,d969d71e544f1541
Normal-281.png,Normal,Normal/images/Normal-281.png,Normal/masks/Normal-281.png,True,G11619,train,3ae42add9618781c545bef9aababaf0f,91baaa38d85e1cd6
Normal-2810.png,Normal,Normal/images/Normal-2810.png,Normal/masks/Normal-2810.png,True,G11620,train,a065dc9bdc0dc2d57060983246317674,d939d6c61a16464f
Normal-2811.png,Normal,Normal/images/Normal-2811.png,Normal/masks/Normal-2811.png,True,G11621,train,5e829d564d3f506f1cd7f208f968f017,882ec35d4f1a4d7c
Normal-2812.png,Normal,Normal/images/Normal-2812.png,Normal/masks/Normal-2812.png,True,G11622,train,e2573d7231f0853003aa9aaec22624fd,dd2c46c35a5d4d45
Normal-2813.png,Normal,Normal/images/Normal-2813.png,Normal/masks/Normal-2813.png,True,G11623,val,e6f2808389b1946aa440cf9e84145eec,897b860a5c037b7d
Normal-2814.png,Normal,Normal/images/Normal-2814.png,Normal/masks/Normal-2814.png,True,G11624,val,53a3efb968908fa61bdb4522d2e45865,c938c65b584f1c9d
Normal-2815.png,Normal,Normal/images/Normal-2815.png,Normal/masks/Normal-2815.png,True,G11625,val,6e874af46a0596d17e467e0d636a1bd1,dd2ec21f58464b59
Normal-2816.png,Normal,Normal/images/Normal-2816.png,Normal/masks/Normal-2816.png,True,G11626,test,1ecc27228a5e46ac6ab07ec428d35ff8,dc2c464f18ccecbc
Normal-2817.png,Normal,Normal/images/Normal-2817.png,Normal/masks/Normal-2817.png,True,G11627,test,95f09dc16125f58f577460536852f8d1,9979861b1c193bec
Normal-2818.png,Normal,Normal/images/Normal-2818.png,Normal/masks/Normal-2818.png,True,G11628,test,cd43efdf9d6b4d7a4a0de6a377413596,dd0a530b5d0f0f1d
Normal-2819.png,Normal,Normal/images/Normal-2819.png,Normal/masks/Normal-2819.png,True,G11629,train,9265ff7ea0dd030f25da28036f27768e,c93d8243bd12ed0f
Normal-282.png,Normal,Normal/images/Normal-282.png,Normal/masks/Normal-282.png,True,G11630,train,3bbbaa1d53d67c68b737395c25ecdf98,812bc65ab44737d6
Normal-2820.png,Normal,Normal/images/Normal-2820.png,Normal/masks/Normal-2820.png,True,G11631,train,4e24a86a00e81b62202323e84516a42f,997f861a1d5b1945
Normal-2821.png,Normal,Normal/images/Normal-2821.png,Normal/masks/Normal-2821.png,True,G11632,train,85d74f33df83b86e82b73c730e8ecc2b,dd2e075b471e014f
Normal-2822.png,Normal,Normal/images/Normal-2822.png,Normal/masks/Normal-2822.png,True,G11633,train,b6fb30eea4b56d7027b3ddc6f7961894,d939061a59e979b8
Normal-2823.png,Normal,Normal/images/Normal-2823.png,Normal/masks/Normal-2823.png,True,G11634,train,1bf28cdee96bc1b43259567ca23340ea,8958165a5f7879e8
Normal-2824.png,Normal,Normal/images/Normal-2824.png,Normal/masks/Normal-2824.png,True,G11635,train,8676631ecaf8079d90db30dad463fcbc,d92e461e186d3cbc
Normal-2825.png,Normal,Normal/images/Normal-2825.png,Normal/masks/Normal-2825.png,True,G11636,train,5c69f07c69664cb6f223bf8fcdf23a39,9d3c465b1b422e3e
Normal-2826.png,Normal,Normal/images/Normal-2826.png,Normal/masks/Normal-2826.png,True,G11637,train,2a3059aa9b254a36cfc3874747baa9bf,c03f42131b5f4f0f
Normal-2827.png,Normal,Normal/images/Normal-2827.png,Normal/masks/Normal-2827.png,True,G11638,train,7fd95228a277af269b97250d505583e3,f979d65658445b18
Normal-2828.png,Normal,Normal/images/Normal-2828.png,Normal/masks/Normal-2828.png,True,G11639,train,acc2e7031dc699ddb28d65c29bfaecee,d53d460b1a6cec39
Normal-2829.png,Normal,Normal/images/Normal-2829.png,Normal/masks/Normal-2829.png,True,G11640,val,58675c28052c9e71e002573360ca975a,c9f9c51a95c79682
Normal-283.png,Normal,Normal/images/Normal-283.png,Normal/masks/Normal-283.png,True,G11641,train,c80685842374ddeb7c56a20731ddd467,913ae65adc591c96
Normal-2830.png,Normal,Normal/images/Normal-2830.png,Normal/masks/Normal-2830.png,True,G11642,train,941cb95ab72298cb58e42cfa66914a56,dc3c8319434fab0f
Normal-2831.png,Normal,Normal/images/Normal-2831.png,Normal/masks/Normal-2831.png,True,G11643,train,7249763349569bb558a526d8e75331ef,d139c2c6169acdc7
Normal-2832.png,Normal,Normal/images/Normal-2832.png,Normal/masks/Normal-2832.png,True,G11644,val,3945539228963abfa1349660282bb65d,897e075a196d79a1
Normal-2833.png,Normal,Normal/images/Normal-2833.png,Normal/masks/Normal-2833.png,True,G11645,val,d2506b5a949fa03525e2ac440aa6907e,dc2f870a5f1d5c44
Normal-2834.png,Normal,Normal/images/Normal-2834.png,Normal/masks/Normal-2834.png,True,G11646,test,fd8094732a620d74dad386f898afceb9,c93b56571a4d0b39
Normal-2835.png,Normal,Normal/images/Normal-2835.png,Normal/masks/Normal-2835.png,True,G11647,test,40f19accdc0a65a230207d3896e868a9,c979c7531a1c5e19
Normal-2836.png,Normal,Normal/images/Normal-2836.png,Normal/masks/Normal-2836.png,True,G11648,train,2d33f4ba71f37d3f4e62b0d64f6f4007,d42e42575f4a2f0e
Normal-2837.png,Normal,Normal/images/Normal-2837.png,Normal/masks/Normal-2837.png,True,G11649,test,fe91833a3fd0b25a056b60db15b16251,817b97c65a191c4f
Normal-2838.png,Normal,Normal/images/Normal-2838.png,Normal/masks/Normal-2838.png,True,G11650,train,e9a9f0f0039956411bdc875ec71a9a16,887b171f1d782b31
Normal-2839.png,Normal,Normal/images/Normal-2839.png,Normal/masks/Normal-2839.png,True,G11651,train,9fcd16c6f6f9149cccfe90498162e869,996d868e185d4d5d
Normal-284.png,Normal,Normal/images/Normal-284.png,Normal/masks/Normal-284.png,True,G11652,train,129f19671855eca03b651f52c8db6389,d176867bcd3520c6
Normal-2840.png,Normal,Normal/images/Normal-2840.png,Normal/masks/Normal-2840.png,True,G11653,val,62d44a9e5442513b66dbd0c412e1b4ac,d93d42c41b1ac74f
Normal-2841.png,Normal,Normal/images/Normal-2841.png,Normal/masks/Normal-2841.png,True,G11654,train,58712bf7e2a8d2ed01ed9a33f0b18723,c93fc65b184e18bc
Normal-2842.png,Normal,Normal/images/Normal-2842.png,Normal/masks/Normal-2842.png,True,G11655,train,e39b400e09b24d9db70b29726fb849b6,9179c60a181f4f5f
Normal-2843.png,Normal,Normal/images/Normal-2843.png,Normal/masks/Normal-2843.png,True,G11656,test,d68700840cd929aa93f9f51582715875,99298618951f5bdb
Normal-2844.png,Normal,Normal/images/Normal-2844.png,Normal/masks/Normal-2844.png,True,G11657,train,327a3846cd9493caf668dd0d56f6798c,d83d420f1d4e2f39
Normal-2845.png,Normal,Normal/images/Normal-2845.png,Normal/masks/Normal-2845.png,True,G11658,train,7898504329c09e6663758d9c58fe388d,d43cf54a93da4293
Normal-2846.png,Normal,Normal/images/Normal-2846.png,Normal/masks/Normal-2846.png,True,G11659,test,69ff1e6a5e84a5c0356970633e81ca52,dd2e421f024f5c6e
Normal-2847.png,Normal,Normal/images/Normal-2847.png,Normal/masks/Normal-2847.png,True,G11660,train,7ee8d982728a5670c68c5e0198f9bc16,dd2d461e1a4c3e78
Normal-2848.png,Normal,Normal/images/Normal-2848.png,Normal/masks/Normal-2848.png,True,G11661,train,4a94a8ffe6cc783a0b46bd4edfb4df19,d85d461a5f1abd0a
Normal-2849.png,Normal,Normal/images/Normal-2849.png,Normal/masks/Normal-2849.png,True,G11662,train,503d33ddf9871dd0b5661be0d732ebdd,dd7c83091f4e3934
Normal-285.png,Normal,Normal/images/Normal-285.png,Normal/masks/Normal-285.png,True,G11663,train,1001342fe30cf320016dc8da71b41585,916cce7ac959584d
Normal-2850.png,Normal,Normal/images/Normal-2850.png,Normal/masks/Normal-2850.png,True,G11664,train,ba33e5beb04cedb2293712f52316f815,91781f835b3958e5
Normal-2851.png,Normal,Normal/images/Normal-2851.png,Normal/masks/Normal-2851.png,True,G11665,train,fdcfd79a107df1e8fe936c2b14547b04,d83fc25b5c481d5c
Normal-2852.png,Normal,Normal/images/Normal-2852.png,Normal/masks/Normal-2852.png,True,G11666,val,66a50a3b3b8796aa326e86c54d341cab,dd3fc2511b024b1f
Normal-2853.png,Normal,Normal/images/Normal-2853.png,Normal/masks/Normal-2853.png,True,G11667,val,262b55de0838f061c79b69074e8c1dfb,d92dc41a1d493eb9
Normal-2854.png,Normal,Normal/images/Normal-2854.png,Normal/masks/Normal-2854.png,True,G11668,test,0840a178580d6d4a8d07e3acf52fba6d,9968061a5b5d7a79
Normal-2855.png,Normal,Normal/images/Normal-2855.png,Normal/masks/Normal-2855.png,True,G11669,train,c4112ed71d8226483b9ff6dbce599a50,897a871a1a4f7978
Normal-2856.png,Normal,Normal/images/Normal-2856.png,Normal/masks/Normal-2856.png,True,G11670,train,f9f54209ff8af42e705e37a23c83b5e2,ccbfc21dc20f904f
Normal-2857.png,Normal,Normal/images/Normal-2857.png,Normal/masks/Normal-2857.png,True,G11671,train,89fb08867d25e965a255aa5660d9a4ea,dd2c500a5f6d743c
Normal-2858.png,Normal,Normal/images/Normal-2858.png,Normal/masks/Normal-2858.png,True,G11672,train,608bfb1459afd2a807e47cb5b86c7b0f,887fc45713c54e1e
Normal-2859.png,Normal,Normal/images/Normal-2859.png,Normal/masks/Normal-2859.png,True,G11673,train,45005914f41501707ad7148cd584b2d7,9129869619ddbeb8
Normal-286.png,Normal,Normal/images/Normal-286.png,Normal/masks/Normal-286.png,True,G11674,train,d4cab32df4d00361fd7af42e2d101cbc,993f561e084b39e3
Normal-2860.png,Normal,Normal/images/Normal-2860.png,Normal/masks/Normal-2860.png,True,G11675,train,2ac62c02d855a850bff83ece047f16f7,dc3cd64b0f454a1d
Normal-2861.png,Normal,Normal/images/Normal-2861.png,Normal/masks/Normal-2861.png,True,G11676,train,2357802967cb1206d5e94a409dac52a0,c92fc61b154f1d4a
Normal-2862.png,Normal,Normal/images/Normal-2862.png,Normal/masks/Normal-2862.png,True,G11677,train,69c3ff3a7a7c07486a09fda7e3e6a0a9,9d2c0e094ee8f9e6
Normal-2863.png,Normal,Normal/images/Normal-2863.png,Normal/masks/Normal-2863.png,True,G11678,train,e14200c16ac102f2b1382497dbce836a,d94a521a183e3cbf
Normal-2864.png,Normal,Normal/images/Normal-2864.png,Normal/masks/Normal-2864.png,True,G11679,train,3c6eae0dc3cfe20986ac5ffa793ab6e8,817f16181a5f7879
Normal-2865.png,Normal,Normal/images/Normal-2865.png,Normal/masks/Normal-2865.png,True,G11680,train,60cb8b82dda474ccfea64f0a1125185c,d92dc61a1ce8b9e4
Normal-2866.png,Normal,Normal/images/Normal-2866.png,Normal/masks/Normal-2866.png,True,G11681,train,a7f3eda50306b4017109045216e8e334,d879d7d6d844861a
Normal-2867.png,Normal,Normal/images/Normal-2867.png,Normal/masks/Normal-2867.png,True,G11682,train,00b23273410ea1c07800a435e65b8f15,99691e5e79e86831
Normal-2868.png,Normal,Normal/images/Normal-2868.png,Normal/masks/Normal-2868.png,True,G11683,val,9d3f68ee8a75563f289f54f50ff746b9,913f0e185d7979c1
Normal-2869.png,Normal,Normal/images/Normal-2869.png,Normal/masks/Normal-2869.png,True,G11684,val,0a1fb58ccc155aa8578b799bf0ae0e79,d93bc61a1b4d781c
Normal-287.png,Normal,Normal/images/Normal-287.png,Normal/masks/Normal-287.png,True,G11685,val,6e5ab13a629c41808f4d16016d330d14,9168c73acb78696c
Normal-2870.png,Normal,Normal/images/Normal-2870.png,Normal/masks/Normal-2870.png,True,G11686,test,dbd1e5053577127bd863f3e1ede79649,dd2e509353554a1f
Normal-2871.png,Normal,Normal/images/Normal-2871.png,Normal/masks/Normal-2871.png,True,G11687,test,9e6b195be2af5b8bde7d67d61904fd91,987f861e145e1a5e
Normal-2872.png,Normal,Normal/images/Normal-2872.png,Normal/masks/Normal-2872.png,True,G11688,test,9b57de9110873d95a21707be43b2b64b,d83d4b1c182b9f93
Normal-2873.png,Normal,Normal/images/Normal-2873.png,Normal/masks/Normal-2873.png,True,G11689,train,359d35699781163ce7e0bfb9df7e25fd,dc2cd24b1f4a0f1d
Normal-2874.png,Normal,Normal/images/Normal-2874.png,Normal/masks/Normal-2874.png,True,G11690,train,e0c6c4c13be4cd79da0bee05927beae7,992d861b4d3a19ee
Normal-2875.png,Normal,Normal/images/Normal-2875.png,Normal/masks/Normal-2875.png,True,G11691,train,0312ef8e6b72345f3db9bf2e45a0791a,992186161f9e7cba
Normal-2876.png,Normal,Normal/images/Normal-2876.png,Normal/masks/Normal-2876.png,True,G11692,train,67b873ed700f88ffdc7046c81b0d1539,dc2ad30f464f4d0e
Normal-2877.png,Normal,Normal/images/Normal-2877.png,Normal/masks/Normal-2877.png,True,G11693,train,2e33557b6dd6e4821b27942d25202530,81798e463a17fe2a
Normal-2878.png,Normal,Normal/images/Normal-2878.png,Normal/masks/Normal-2878.png,True,G11694,train,95b18d0a13ec97f2c61cd39c16437969,d03d860a1d0f6f3d
Normal-2879.png,Normal,Normal/images/Normal-2879.png,Normal/masks/Normal-2879.png,True,G11695,train,9f5f78a57c4474062be21586757b9baf,dc2d421f440f5f4d
Normal-288.png,Normal,Normal/images/Normal-288.png,Normal/masks/Normal-288.png,True,G11696,train,605c3f95bf3d9e23dce5547c17822f13,917e8f1a0d185cd7
Normal-2880.png,Normal,Normal/images/Normal-2880.png,Normal/masks/Normal-2880.png,True,G11697,val,85de0cf2bacdc6ad96684c7dbfbde407,fc2ed31b045d444f
Normal-2881.png,Normal,Normal/images/Normal-2881.png,Normal/masks/Normal-2881.png,True,G11698,train,fae3d5d2a7f174f182b554baef76613d,9c9c6363cedc6868
Normal-2882.png,Normal,Normal/images/Normal-2882.png,Normal/masks/Normal-2882.png,True,G11699,val,09060addd1367ff80bef1c139369885d,dd4e474b0d0e0f0f
Normal-2883.png,Normal,Normal/images/Normal-2883.png,Normal/masks/Normal-2883.png,True,G11700,train,2ed1f35e93c864c3b7a85cdb2da312df,993dc2d3dbc2c390
Normal-2884.png,Normal,Normal/images/Normal-2884.png,Normal/masks/Normal-2884.png,True,G11701,val,5a81a7da35572c08c9e6ef700bce89ca,c90d061e5d5a5d5d
Normal-2885.png,Normal,Normal/images/Normal-2885.png,Normal/masks/Normal-2885.png,True,G11702,train,f7f72992d7754b72d901d561ac04a8b3,813d061e1d7f7a68
Normal-2886.png,Normal,Normal/images/Normal-2886.png,Normal/masks/Normal-2886.png,True,G11703,train,a5100d7e19670a7d9f0867e8f20e61cd,dd2c820b0dea39f5
Normal-2887.png,Normal,Normal/images/Normal-2887.png,Normal/masks/Normal-2887.png,True,G11704,train,d8afa2ac4fe0855b1c62aa8cde207092,d42f435d0e2a7cac
Normal-2888.png,Normal,Normal/images/Normal-2888.png,Normal/masks/Normal-2888.png,True,G11705,train,18b353da2ba54f9dd2df4cfc8a023758,d94e47190d0e1f4f
Normal-2889.png,Normal,Normal/images/Normal-2889.png,Normal/masks/Normal-2889.png,True,G11706,train,22d3577855ea2e2d5e81dbd53905cc5b,dd3d421a554b1e4b
Normal-289.png,Normal,Normal/images/Normal-289.png,Normal/masks/Normal-289.png,True,G11707,test,7efce324b9d73c2ad9c7fb469b9e8a81,957e8f1a8d0d4c4d
Normal-2890.png,Normal,Normal/images/Normal-2890.png,Normal/masks/Normal-2890.png,True,G11708,test,f12607c27a7652aef5f9e1c6acd10a06,dd2cd257d2435d50
Normal-2891.png,Normal,Normal/images/Normal-2891.png,Normal/masks/Normal-2891.png,True,G11709,test,f742059ab02bea97a2f9007253b050c8,dc2d434f0c4f2cec
Normal-2892.png,Normal,Normal/images/Normal-2892.png,Normal/masks/Normal-2892.png,True,G11710,val,62b1c0eac65eba7aa7d7e4a6d3f24fc2,d93b479a1b09ee18
Normal-2893.png,Normal,Normal/images/Normal-2893.png,Normal/masks/Normal-2893.png,True,G11711,train,0f4993300d68cb1e0fa26f68ead1eccd,886b421a5d0f1f3f
Normal-2894.png,Normal,Normal/images/Normal-2894.png,Normal/masks/Normal-2894.png,True,G11712,train,4d1ae7284ae5f364a547ebb0848f19f4,d93d46171a445e1f
Normal-2895.png,Normal,Normal/images/Normal-2895.png,Normal/masks/Normal-2895.png,True,G11713,train,a4f03838b208bbe15f46a6060a4cc255,992d1e1c48e9f8e5
Normal-2896.png,Normal,Normal/images/Normal-2896.png,Normal/masks/Normal-2896.png,True,G11714,test,b82b5475562dc92c503cc756248fd2bc,997fc612174e1a47
Normal-2897.png,Normal,Normal/images/Normal-2897.png,Normal/masks/Normal-2897.png,True,G11715,train,7731ecc2a42149692c360531a9cdb90b,9941161a595b1f5f
Normal-2898.png,Normal,Normal/images/Normal-2898.png,Normal/masks/Normal-2898.png,True,G11716,train,82e9ea99137e6ffa2fa7a237003c20fe,917d061a1d5b195f
Normal-2899.png,Normal,Normal/images/Normal-2899.png,Normal/masks/Normal-2899.png,True,G11717,train,19145f0e87480ea42875cfc6dc1a98e6,9969c61ad55f184d
Normal-29.png,Normal,Normal/images/Normal-29.png,Normal/masks/Normal-29.png,True,G11718,train,54414f2cdacc30a2f12832045c338361,c979470a1ccb1b9d
Normal-290.png,Normal,Normal/images/Normal-290.png,Normal/masks/Normal-290.png,True,G11719,train,6d34fa3d108428cc27e2aabc449d892e,d46ea37be16468e0
Normal-2900.png,Normal,Normal/images/Normal-2900.png,Normal/masks/Normal-2900.png,True,G11720,train,480a1626831e88e764881dfef0a13255,d979c41e1bc51b19
Normal-2901.png,Normal,Normal/images/Normal-2901.png,Normal/masks/Normal-2901.png,True,G11721,train,1c68bad3f8fd0c02268c92ed3a81cb3b,dd2c531f1c4e3d48
Normal-2902.png,Normal,Normal/images/Normal-2902.png,Normal/masks/Normal-2902.png,True,G11722,train,297d113da8d1f5202c14436450327fcc,d96d061ac71b154f
Normal-2903.png,Normal,Normal/images/Normal-2903.png,Normal/masks/Normal-2903.png,True,G11723,train,2d65b4a695b8cc9c5ecd93a54a693fa3,892d065879ece9b9
Normal-2904.png,Normal,Normal/images/Normal-2904.png,Normal/masks/Normal-2904.png,True,G11724,train,c4353dd619fb5d4393cc10c6de3f3d84,c92c060e1d5d7a7d
Normal-2905.png,Normal,Normal/images/Normal-2905.png,Normal/masks/Normal-2905.png,True,G11725,train,1d8c7b9ff008c335c221658c457b05e4,d46f47080d6e2d2f
Normal-2906.png,Normal,Normal/images/Normal-2906.png,Normal/masks/Normal-2906.png,True,G11726,test,8624359c6f36a8fb013a4ef38cb23436,9168061a5e7e78f9
Normal-2907.png,Normal,Normal/images/Normal-2907.png,Normal/masks/Normal-2907.png,True,G11727,val,2381fb90028ce1f172684cefd14a1d94,d57d430b19ae2d34
Normal-2908.png,Normal,Normal/images/Normal-2908.png,Normal/masks/Normal-2908.png,True,G11728,val,60e86f56fa785b013039c68377cd4f45,b97b861a14471b1f
Normal-2909.png,Normal,Normal/images/Normal-2909.png,Normal/masks/Normal-2909.png,True,G11729,train,f632b1d7da2eac3f7dc4b6fd428eaa2f,d93b141e1a593b59
Normal-291.png,Normal,Normal/images/Normal-291.png,Normal/masks/Normal-291.png,True,G11730,train,c154032eb93f188e3c4dbd4c57f90675,98a9c71ed64b58c6
Normal-2910.png,Normal,Normal/images/Normal-2910.png,Normal/masks/Normal-2910.png,True,G11731,test,46b42edd2933675ad87f5228707f02fc,dc2ec25b0f0e683d
Normal-2911.png,Normal,Normal/images/Normal-2911.png,Normal/masks/Normal-2911.png,True,G11732,test,e9ad954c34aa7e91ad32ea54ea42664a,897dc6861a5dea58
Normal-2912.png,Normal,Normal/images/Normal-2912.png,Normal/masks/Normal-2912.png,True,G11733,val,799c1af800c4f138fca007ae71777b60,99288617d91efa59
Normal-2913.png,Normal,Normal/images/Normal-2913.png,Normal/masks/Normal-2913.png,True,G11734,train,56384c37c6d6bb5b7e517013c2ebbc0b,817856071a1fed3e
Normal-2914.png,Normal,Normal/images/Normal-2914.png,Normal/masks/Normal-2914.png,True,G11735,train,f4dd1b4f4ce4619301a00681f0fd5525,886847575b975e1a
Normal-2915.png,Normal,Normal/images/Normal-2915.png,Normal/masks/Normal-2915.png,True,G11736,train,d623be7a1ead1911a5f327c05cb22020,c87bc51696c71e94
Normal-2916.png,Normal,Normal/images/Normal-2916.png,Normal/masks/Normal-2916.png,True,G11737,test,b499171a73c4951a2224050e68ef7675,997bc61e1c475a38
Normal-2917.png,Normal,Normal/images/Normal-2917.png,Normal/masks/Normal-2917.png,True,G11738,train,3bf58850548ee0fa8023a598cef67f43,9929045e5a7df8e8
Normal-2918.png,Normal,Normal/images/Normal-2918.png,Normal/masks/Normal-2918.png,True,G11739,train,4af2f36a04ea619ecadd8ca40192ab2e,9179c61e39e5a1b1
Normal-2919.png,Normal,Normal/images/Normal-2919.png,Normal/masks/Normal-2919.png,True,G11740,train,65ca89aa1161b3b1c928c36ecf5f1891,d42cc3531d4f473a
Normal-292.png,Normal,Normal/images/Normal-292.png,Normal/masks/Normal-292.png,True,G11741,train,b4ee4d7a0cc4f0d2cf5671aeff0b2d55,816f8f5b41454bc7
Normal-2920.png,Normal,Normal/images/Normal-2920.png,Normal/masks/Normal-2920.png,True,G11742,train,5d1d628e5a9ccc83f73a39abef2111fa,dc2d430f4f0f2cac
Normal-2921.png,Normal,Normal/images/Normal-2921.png,Normal/masks/Normal-2921.png,True,G11743,train,69929ea06051ac38e15fc99bdf141cfe,d13b421e1c2d3d3d
Normal-2922.png,Normal,Normal/images/Normal-2922.png,Normal/masks/Normal-2922.png,True,G11744,train,dc8b7fc0d873ee04917a18c287247f90,dc2dc2531f424f1d
Normal-2923.png,Normal,Normal/images/Normal-2923.png,Normal/masks/Normal-2923.png,True,G11745,train,daf05677f70c7fe9140a434eaff8067b,9939061e1c5c5b7d
Normal-2924.png,Normal,Normal/images/Normal-2924.png,Normal/masks/Normal-2924.png,True,G11746,train,6a2c865c5a55d7b7df62a1c670f410f0,8178961e1a295f7b
Normal-2925.png,Normal,Normal/images/Normal-2925.png,Normal/masks/Normal-2925.png,True,G11747,val,0bfb4352d4625d6d5f686466d8a20f03,9d68070b094f1f5f
Normal-2926.png,Normal,Normal/images/Normal-2926.png,Normal/masks/Normal-2926.png,True,G11748,val,ad2d8d519304e26122def34e06c2eafd,907a67c73bc92ea0
Normal-2927.png,Normal,Normal/images/Normal-2927.png,Normal/masks/Normal-2927.png,True,G11749,test,b1bb3d6734cdb403ca0b8bc11cc9556a,9928161a4f1e1f5f
Normal-2928.png,Normal,Normal/images/Normal-2928.png,Normal/masks/Normal-2928.png,True,G11750,train,be08ad1412535bcc252d6a90a6a35a64,9968871a1f4e18df
Normal-2929.png,Normal,Normal/images/Normal-2929.png,Normal/masks/Normal-2929.png,True,G11751,train,ee3c5714223ba6cecdeb2ef665da0dbb,996986171b4f0e3b
Normal-293.png,Normal,Normal/images/Normal-293.png,Normal/masks/Normal-293.png,True,G11752,train,b1c9c7fa300bdf26193db8310db1c6ee,c129e27ada5e7854
Normal-2930.png,Normal,Normal/images/Normal-2930.png,Normal/masks/Normal-2930.png,True,G11753,train,b8ea5c1d82460c6f5bf0226ab04548fd,917ec756184d4c3d
Normal-2931.png,Normal,Normal/images/Normal-2931.png,Normal/masks/Normal-2931.png,True,G11754,train,f7cca0c07193f88921dd13a21747312d,d52a52574b544f4e
Normal-2932.png,Normal,Normal/images/Normal-2932.png,Normal/masks/Normal-2932.png,True,G11755,train,b06c2003a7748b5be3783af423063b0e,997dc61a1c4c3bb8
Normal-2933.png,Normal,Normal/images/Normal-2933.png,Normal/masks/Normal-2933.png,True,G11756,train,419d6ad68e5e203d19d861027b0e5716,8879c7c69e16c64b
Normal-2934.png,Normal,Normal/images/Normal-2934.png,Normal/masks/Normal-2934.png,True,G11757,train,1d794ed474c84e3f409e095d02697951,c13c07d353463b1f
Normal-2935.png,Normal,Normal/images/Normal-2935.png,Normal/masks/Normal-2935.png,True,G11758,train,7d09f2fbc5e13d63dfcb6868917fd198,dd3bd20f5f091c09
Normal-2936.png,Normal,Normal/images/Normal-2936.png,Normal/masks/Normal-2936.png,True,G11759,val,f530a15ec0df8df76878bbd7c7cc9ecc,d92bc616df06c712
Normal-2937.png,Normal,Normal/images/Normal-2937.png,Normal/masks/Normal-2937.png,True,G11760,val,48d31bb4280890ca891944d1ec58944a,9978861a59ef3aa1
Normal-2938.png,Normal,Normal/images/Normal-2938.png,Normal/masks/Normal-2938.png,True,G11761,train,45288d0575b61fb330e1d5e2f9359d13,dd3c524b0f5d4c58
Normal-2939.png,Normal,Normal/images/Normal-2939.png,Normal/masks/Normal-2939.png,True,G11762,val,05c5fe3d03b6b5effc36e345e5e5382e,d03dc65f190d293d
Normal-294.png,Normal,Normal/images/Normal-294.png,Normal/masks/Normal-294.png,True,G11763,test,05ac0a8e4b7c000e3d52c1bc516a8c34,91e8ce78d9393925
Normal-2940.png,Normal,Normal/images/Normal-2940.png,Normal/masks/Normal-2940.png,True,G11764,train,9d4c17011ac7517981914225e251d9a8,9968c61a194f7ab9
Normal-2941.png,Normal,Normal/images/Normal-2941.png,Normal/masks/Normal-2941.png,True,G11765,test,a5c000f345eaa9b15b9ef40967878eeb,cc3d43c21f4e6c2d
Normal-2942.png,Normal,Normal/images/Normal-2942.png,Normal/masks/Normal-2942.png,True,G11766,train,a67bb91a05385f068a662409110f4fea,9171fe1af9a105e1
Normal-2943.png,Normal,Normal/images/Normal-2943.png,Normal/masks/Normal-2943.png,True,G11767,train,8745eb80fe8d1d002357c41721453537,d92a061f465f1a4f
Normal-2944.png,Normal,Normal/images/Normal-2944.png,Normal/masks/Normal-2944.png,True,G11768,train,08c480851c79ec6b1e5583dd53f3eb50,d03c0f83909bcfce
Normal-2945.png,Normal,Normal/images/Normal-2945.png,Normal/masks/Normal-2945.png,True,G11769,train,567aab0d8b97d2f3d9aebba279f4c1b7,993cc64b13c71f1a
Normal-2946.png,Normal,Normal/images/Normal-2946.png,Normal/masks/Normal-2946.png,True,G11770,train,62ab78948a807fe097d5899f9e2a65e5,8939d61f94471d5a
Normal-2947.png,Normal,Normal/images/Normal-2947.png,Normal/masks/Normal-2947.png,True,G11771,train,f8f091ce7b3eea7b838bf512acfb35ff,917806161b7d6a7b
Normal-2948.png,Normal,Normal/images/Normal-2948.png,Normal/masks/Normal-2948.png,True,G11772,train,c7bf45df13cea8887661b09a651ecee0,85317aca7894e667
Normal-2949.png,Normal,Normal/images/Normal-2949.png,Normal/masks/Normal-2949.png,True,G11773,test,d28bf813a6f0d989e12e17646e18b049,8836021f471b5f5f
Normal-295.png,Normal,Normal/images/Normal-295.png,Normal/masks/Normal-295.png,True,G11774,test,34727b6ac50ce7d13672d72c53735811,813be71e3a945939
Normal-2950.png,Normal,Normal/images/Normal-2950.png,Normal/masks/Normal-2950.png,True,G11775,train,4e6cd8461d3baa6c3ff06e62e8bb7b4e,997d061e551b194f
Normal-2951.png,Normal,Normal/images/Normal-2951.png,Normal/masks/Normal-2951.png,True,G11776,train,c78c49387776c242489119d9246a8a63,9179461a1e5f3ac5
Normal-2952.png,Normal,Normal/images/Normal-2952.png,Normal/masks/Normal-2952.png,True,G11777,train,3d4d091bbd6993c8e6839be0238f4eef,897d861a855f194f
Normal-2953.png,Normal,Normal/images/Normal-2953.png,Normal/masks/Normal-2953.png,True,G11778,train,184865848d92569d0d302e93d5283e81,dd2ed2935bd1420b
Normal-2954.png,Normal,Normal/images/Normal-2954.png,Normal/masks/Normal-2954.png,True,G11779,train,41c36d4814b49fbe26808b1013ff8222,883d47d21dcd1a9d
Normal-2955.png,Normal,Normal/images/Normal-2955.png,Normal/masks/Normal-2955.png,True,G11780,train,55739b5b6739ca78181c4f6bf1eed93d,897f161a1b057ab9
Normal-2956.png,Normal,Normal/images/Normal-2956.png,Normal/masks/Normal-2956.png,True,G11781,train,4cba89af145db64f93b01b2cca45b197,997f061a192a5bba
Normal-2957.png,Normal,Normal/images/Normal-2957.png,Normal/masks/Normal-2957.png,True,G11782,train,5bc541b534f78c368826fd93fafcbffa,995907161a59697f
Normal-2958.png,Normal,Normal/images/Normal-2958.png,Normal/masks/Normal-2958.png,True,G11783,train,377ac94a2376554d61d07f344537c26d,996f0612471e175f
Normal-2959.png,Normal,Normal/images/Normal-2959.png,Normal/masks/Normal-2959.png,True,G11784,test,bad0e0f790549cf28b32497cfd9e436f,996f861a161b1c5f
Normal-296.png,Normal,Normal/images/Normal-296.png,Normal/masks/Normal-296.png,True,G11785,train,90801803d644f728741ba47008f9f46d,9074865bab9d6995
Normal-2960.png,Normal,Normal/images/Normal-2960.png,Normal/masks/Normal-2960.png,True,G11786,val,422e0f23e5d486d9a573afcfb537861f,9969041a5f7b61cd
Normal-2961.png,Normal,Normal/images/Normal-2961.png,Normal/masks/Normal-2961.png,True,G11787,train,2c0136423fda1a6159f074d8e59b87cc,dc2dc25b464a1f4d
Normal-2962.png,Normal,Normal/images/Normal-2962.png,Normal/masks/Normal-2962.png,True,G11788,val,feb25f532e5834eeeb8b9163a93ed751,997947161a594d1f
Normal-2963.png,Normal,Normal/images/Normal-2963.png,Normal/masks/Normal-2963.png,True,G11789,val,6a2626a9d023712df0859b7005b29465,dd2cd7030f4c4a5b
Normal-2964.png,Normal,Normal/images/Normal-2964.png,Normal/masks/Normal-2964.png,True,G11790,test,ae5c376b5895a89edd572e7d1e59f7a6,f825c61247571d4f
Normal-2965.png,Normal,Normal/images/Normal-2965.png,Normal/masks/Normal-2965.png,True,G11791,train,b36218daa1d190cc091bcd2e2e097c66,996c865b954f1ac5
Normal-2966.png,Normal,Normal/images/Normal-2966.png,Normal/masks/Normal-2966.png,True,G11792,test,ea9af1848ed686e3520d8ecdd15e094b,997d061e485b39e9
Normal-2967.png,Normal,Normal/images/Normal-2967.png,Normal/masks/Normal-2967.png,True,G11793,train,c206eae747ad1e9394e370fbf8a63632,9d2e061b0d5e5c5e
Normal-2968.png,Normal,Normal/images/Normal-2968.png,Normal/masks/Normal-2968.png,True,G11794,train,bed96fc914e7d0f8c14bb9d056404f25,9929161e1c5f5e1d
Normal-2969.png,Normal,Normal/images/Normal-2969.png,Normal/masks/Normal-2969.png,True,G11795,train,0390848b37085ba31c5f8672b07a0d97,d42f42514b4f2e2f
Normal-297.png,Normal,Normal/images/Normal-297.png,Normal/masks/Normal-297.png,True,G11796,train,dc1dca4d45788830e2aa2a56b48f0286,957e8f1e890c58cd
Normal-2970.png,Normal,Normal/images/Normal-2970.png,Normal/masks/Normal-2970.png,True,G11797,test,11d59162714329a78319881110b4e200,c03d42571f5a1f0f
Normal-2971.png,Normal,Normal/images/Normal-2971.png,Normal/masks/Normal-2971.png,True,G11798,train,3953c670be132ad5a7673f046f6b8b78,887d425b5b4f0f0e
Normal-2972.png,Normal,Normal/images/Normal-2972.png,Normal/masks/Normal-2972.png,True,G11799,train,b8860c39e30efa525a3d3ec4772a825c,993d461b1a4f1e9a
Normal-2973.png,Normal,Normal/images/Normal-2973.png,Normal/masks/Normal-2973.png,True,G11800,test,8adec3abb1a07284b88033b36242ce5e,9979c61a594a5b3c
Normal-2974.png,Normal,Normal/images/Normal-2974.png,Normal/masks/Normal-2974.png,True,G11801,train,a45d0e2b7a93e86495ff55eb74d5f69f,d43ec25b0f4a2f0e
Normal-2975.png,Normal,Normal/images/Normal-2975.png,Normal/masks/Normal-2975.png,True,G11802,train,71d2514f5d48d8cfc0244cc14e0d9c10,996817121a5f787b
Normal-2976.png,Normal,Normal/images/Normal-2976.png,Normal/masks/Normal-2976.png,True,G11803,train,01f5c6e55edcdcc8f6f9c8d8d7fd7f10,c85d061e184dedbd
Normal-2977.png,Normal,Normal/images/Normal-2977.png,Normal/masks/Normal-2977.png,True,G11804,train,cc617e8b32d1d5bd85657953acf7948b,987d861a1f5d5a58
Normal-2978.png,Normal,Normal/images/Normal-2978.png,Normal/masks/Normal-2978.png,True,G11805,train,97ac2ee5a104bf89334dbafb565ad7f5,c13d460d195f6d39
Normal-2979.png,Normal,Normal/images/Normal-2979.png,Normal/masks/Normal-2979.png,True,G11806,train,fadec3d590fa39e0559d6ea8acc9e49c,dc2c420b1cacfcbd
Normal-298.png,Normal,Normal/images/Normal-298.png,Normal/masks/Normal-298.png,True,G11807,train,bac767b980d79ce580480361556b715a,907b871ae939f01d
Normal-2980.png,Normal,Normal/images/Normal-2980.png,Normal/masks/Normal-2980.png,True,G11808,train,8a8de6bdbdc71677b40284c838badf61,dc2e424b0f455f1d
Normal-2981.png,Normal,Normal/images/Normal-2981.png,Normal/masks/Normal-2981.png,True,G11809,train,1690afb5c24cd5774db7d07d7a4ccd0b,d17b56125a554b1b
Normal-2982.png,Normal,Normal/images/Normal-2982.png,Normal/masks/Normal-2982.png,True,G11810,train,211641650c50fb1a9198aa7aa9eca282,c93ed6431b1e4a4b
Normal-2983.png,Normal,Normal/images/Normal-2983.png,Normal/masks/Normal-2983.png,True,G11811,val,24c8960a9e4486dc6654987d56892812,dd3c520b0aceab99
Normal-2984.png,Normal,Normal/images/Normal-2984.png,Normal/masks/Normal-2984.png,True,G11812,test,7dc0e88cad943984671ed3365261a927,992d861a554f1bc7
Normal-2985.png,Normal,Normal/images/Normal-2985.png,Normal/masks/Normal-2985.png,True,G11813,train,701ab6e3b70a377e2405d8fbd0f6a72b,8978454a1f0d7e3d
Normal-2986.png,Normal,Normal/images/Normal-2986.png,Normal/masks/Normal-2986.png,True,G11814,train,6c5a2a395fff64010c9f36cc6b1f25cc,9968061e1b5f398f
Normal-2987.png,Normal,Normal/images/Normal-2987.png,Normal/masks/Normal-2987.png,True,G11815,val,b308c7d6901313b7759f0649bdcfe728,807fc6d2184acfba
Normal-2988.png,Normal,Normal/images/Normal-2988.png,Normal/masks/Normal-2988.png,True,G11816,val,7dbbfecd0ccaa14a6e1e7b77edea1da6,cd6cc61a5d6c38b4
Normal-2989.png,Normal,Normal/images/Normal-2989.png,Normal/masks/Normal-2989.png,True,G11817,train,915d3e2101e7f7f7a1adc21d68bfe9a1,d92fc6121e4d6a39
Normal-299.png,Normal,Normal/images/Normal-299.png,Normal/masks/Normal-299.png,True,G11818,train,8b18cc791af666ffc44c465e548bf8e2,d1feaf0b89012dc6
Normal-2990.png,Normal,Normal/images/Normal-2990.png,Normal/masks/Normal-2990.png,True,G11819,train,6288159862262d59b28cf20e60a4feb2,d9ad26c239d26b16
Normal-2991.png,Normal,Normal/images/Normal-2991.png,Normal/masks/Normal-2991.png,True,G11820,train,97e0dc78aab35ae7d9a3999a6a3f20b5,d83dc20f1a0f3d99
Normal-2992.png,Normal,Normal/images/Normal-2992.png,Normal/masks/Normal-2992.png,True,G11821,train,18ea14beb82e5a389e1d5cdbbcc0eb00,997b041a5f097a79
Normal-2993.png,Normal,Normal/images/Normal-2993.png,Normal/masks/Normal-2993.png,True,G11822,train,078bcf8eb5f1123389315009c27de12e,c979c60e1a4d4f39
Normal-2994.png,Normal,Normal/images/Normal-2994.png,Normal/masks/Normal-2994.png,True,G11823,train,e1b752b31213c66e2e4629318b680295,dc3d421f8f0839e5
Normal-2995.png,Normal,Normal/images/Normal-2995.png,Normal/masks/Normal-2995.png,True,G11824,train,3ee3c740d589be8fbdbb82df9cbef141,817f86da5941f5c2
Normal-2996.png,Normal,Normal/images/Normal-2996.png,Normal/masks/Normal-2996.png,True,G11825,test,820fa70330a6feda797b9e5e48d131e6,997fc61a590f184b
Normal-2997.png,Normal,Normal/images/Normal-2997.png,Normal/masks/Normal-2997.png,True,G11826,train,2579b594b6b22ba7331198c9d0e3f669,817b4218393f3d3d
Normal-2998.png,Normal,Normal/images/Normal-2998.png,Normal/masks/Normal-2998.png,True,G11827,train,8c4ece4569a750faa4420050c7ebaa97,dd2b521f480f1d4d
Normal-2999.png,Normal,Normal/images/Normal-2999.png,Normal/masks/Normal-2999.png,True,G11828,val,e4a6b420a5f009373768a48df1374f9a,997985161a5d7a78
Normal-3.png,Normal,Normal/images/Normal-3.png,Normal/masks/Normal-3.png,True,G11829,val,9a311f20f62b0aeae644f60242376963,817f070e195d5f58
Normal-30.png,Normal,Normal/images/Normal-30.png,Normal/masks/Normal-30.png,True,G11830,val,fa9fe16b2baeebb332d025f88eb60b08,c13dc61b554f1e49
Normal-300.png,Normal,Normal/images/Normal-300.png,Normal/masks/Normal-300.png,True,G11831,test,294ef6a673afea0e08940b384964f052,9178db389d189ad6
Normal-3000.png,Normal,Normal/images/Normal-3000.png,Normal/masks/Normal-3000.png,True,G11832,test,cdaf1b75bfdc524ce4670584e81e81a5,d93b825f184b3d1a
Normal-3001.png,Normal,Normal/images/Normal-3001.png,Normal/masks/Normal-3001.png,True,G11833,train,2258e8f707dc27832bb968dc13b229ca,cc3d4352594bc34f
Normal-3002.png,Normal,Normal/images/Normal-3002.png,Normal/masks/Normal-3002.png,True,G11834,train,8a849b5e9a940651dd776bf397441502,dd2ad21f540a7963
Normal-3003.png,Normal,Normal/images/Normal-3003.png,Normal/masks/Normal-3003.png,True,G11835,train,7b7ebe987ad1791698a5d3ebbc644b74,dd3cd60b5d421b0d
Normal-3004.png,Normal,Normal/images/Normal-3004.png,Normal/masks/Normal-3004.png,True,G11836,train,788677f87ed22a39f74f32b60cc86e7f,913b07161e155f1f
Normal-3005.png,Normal,Normal/images/Normal-3005.png,Normal/masks/Normal-3005.png,True,G11837,train,103c5afc40251442a86febe4f755bb09,d93bd2171e062f19
Normal-3006.png,Normal,Normal/images/Normal-3006.png,Normal/masks/Normal-3006.png,True,G11838,train,f6deb7ba6991d154c3871e83ddc8232f,992d861a586dfa78
Normal-3007.png,Normal,Normal/images/Normal-3007.png,Normal/masks/Normal-3007.png,True,G11839,train,d2fc3bc7b43ce662349949095131b2e9,d03dc34b3c0b2d3d
Normal-3008.png,Normal,Normal/images/Normal-3008.png,Normal/masks/Normal-3008.png,True,G11840,train,46594b334f0248f3d83d1f5bab5ddc3b,9978c6061b174f3b
Normal-3009.png,Normal,Normal/images/Normal-3009.png,Normal/masks/Normal-3009.png,True,G11841,test,a1d059a3f2a3d9befa20f73d0ed43633,d87bd5c612434f19
Normal-301.png,Normal,Normal/images/Normal-301.png,Normal/masks/Normal-301.png,True,G11842,val,4f9a22cfae769395bca14f56a8aeed5a,d07dc71a9b693858
Normal-3010.png,Normal,Normal/images/Normal-3010.png,Normal/masks/Normal-3010.png,True,G11843,train,fa12e8150e32e98c58f6c5c35d7b1581,992f061a055e5e5f
Normal-3011.png,Normal,Normal/images/Normal-3011.png,Normal/masks/Normal-3011.png,True,G11844,test,7626a064730e6ebae16c4f903e5763f1,9568005e5a1bdf4f
Normal-3012.png,Normal,Normal/images/Normal-3012.png,Normal/masks/Normal-3012.png,True,G11845,train,8f811b52ce1a150842397d2cf62a9ce1,dc2e434b0c4c6cbf
Normal-3013.png,Normal,Normal/images/Normal-3013.png,Normal/masks/Normal-3013.png,True,G11846,train,3fb1358bd76c16a40c5d7989bffca24a,dd2b560e1a58e9e8
Normal-3014.png,Normal,Normal/images/Normal-3014.png,Normal/masks/Normal-3014.png,True,G11847,train,f740b86273a8d69d87469e2c995609fd,993d061e5a454f79
Normal-3015.png,Normal,Normal/images/Normal-3015.png,Normal/masks/Normal-3015.png,True,G11848,train,2f37576407ef001087fadffc0bf479b3,dd3dc20b154f1e58
Normal-3016.png,Normal,Normal/images/Normal-3016.png,Normal/masks/Normal-3016.png,True,G11849,train,fc5c9c9cb3252b50320a01e8a6fdb722,816e065b4d5f4b69
Normal-3017.png,Normal,Normal/images/Normal-3017.png,Normal/masks/Normal-3017.png,True,G11850,val,52c5ceef1b62364e91d33aca3e7af250,997b471a194f1b0e
Normal-3018.png,Normal,Normal/images/Normal-3018.png,Normal/masks/Normal-3018.png,True,G11851,val,fbcbdf8dd47583dc1ec48c4d068f7c50,c93b4615164e1f5e
Normal-3019.png,Normal,Normal/images/Normal-3019.png,Normal/masks/Normal-3019.png,True,G11852,test,ee6be4c1bed13fa7040b3db9ef183e42,991d061e1e494f7e
Normal-302.png,Normal,Normal/images/Normal-302.png,Normal/masks/Normal-302.png,True,G11853,train,c315d029110306d52cd4d906c6965d79,9129c21ad8693dfb
Normal-3020.png,Normal,Normal/images/Normal-3020.png,Normal/masks/Normal-3020.png,True,G11854,test,dbbcfcb1807cdb7667f791745ef93f46,9978061a5bcd7a69
Normal-3021.png,Normal,Normal/images/Normal-3021.png,Normal/masks/Normal-3021.png,True,G11855,train,c220bb042507620afb9b5e76f2a2b440,a929069657971f96
Normal-3022.png,Normal,Normal/images/Normal-3022.png,Normal/masks/Normal-3022.png,True,G11856,train,f880829f5037b7c6d0355e2d303411a3,d569c75b1b461c19
Normal-3023.png,Normal,Normal/images/Normal-3023.png,Normal/masks/Normal-3023.png,True,G11857,train,e2586dc11b7a7cf93d97b2b94dbffa11,d42e035f4d4e7c4c
Normal-3024.png,Normal,Normal/images/Normal-3024.png,Normal/masks/Normal-3024.png,True,G11858,test,dd9b24e73f3ed684731287d329f7f329,d93bd60b1d081bbc
Normal-3025.png,Normal,Normal/images/Normal-3025.png,Normal/masks/Normal-3025.png,True,G11859,train,23bcc49329dae4d2f3c4d5c54ceb9418,9961161f4d1e4b3d
Normal-3026.png,Normal,Normal/images/Normal-3026.png,Normal/masks/Normal-3026.png,True,G11860,train,21715ecb8974f8d664732f31467c79fe,d93bc61b194c39ac
Normal-3027.png,Normal,Normal/images/Normal-3027.png,Normal/masks/Normal-3027.png,True,G11861,train,4cb119ce754ac1ac5bc7910805d79f94,936bd4945a1b0f3a
Normal-3028.png,Normal,Normal/images/Normal-3028.png,Normal/masks/Normal-3028.png,True,G11862,train,651f04be2c79e49472056b7e298497e2,dd6ad24b4e4a4d0e
Normal-3029.png,Normal,Normal/images/Normal-3029.png,Normal/masks/Normal-3029.png,True,G11863,train,5fa9e90a217a06258427064f2716c5ad,99ba1a194a196fd9
Normal-303.png,Normal,Normal/images/Normal-303.png,Normal/masks/Normal-303.png,True,G11864,train,e2db029f7fe307d3e3507cc6e1d42e86,916cc75abcee2894
Normal-3030.png,Normal,Normal/images/Normal-3030.png,Normal/masks/Normal-3030.png,True,G11865,train,e7a543cca900fa0bb2cb3e1e4962ef4f,d93dc61b1d46191d
Normal-3031.png,Normal,Normal/images/Normal-3031.png,Normal/masks/Normal-3031.png,True,G11866,train,d8ff2202264f56d06157281725dd0414,d93fc6029b0afe09
Normal-3032.png,Normal,Normal/images/Normal-3032.png,Normal/masks/Normal-3032.png,True,G11867,train,9f463facbb88e6be3249ea7ae2092cb0,9969471e196d3cb1
Normal-3033.png,Normal,Normal/images/Normal-3033.png,Normal/masks/Normal-3033.png,True,G11868,train,9103ad84600546bd2a8f390903b934cf,bc09431f4a3f426f
Normal-3034.png,Normal,Normal/images/Normal-3034.png,Normal/masks/Normal-3034.png,True,G11869,val,c26480c04185977a791301f7153a8bf1,c1292e567fe426b2
Normal-3035.png,Normal,Normal/images/Normal-3035.png,Normal/masks/Normal-3035.png,True,G11870,val,d338795dee39eb0e68e9cd2cf19c09ff,9169c65619491f3f
Normal-3036.png,Normal,Normal/images/Normal-3036.png,Normal/masks/Normal-3036.png,True,G11871,train,1c2a141c2b39fd8def02da35e91534fd,c938861b154f3bad
Normal-3037.png,Normal,Normal/images/Normal-3037.png,Normal/masks/Normal-3037.png,True,G11872,val,94abbc46b3cb0d66656101ac758aba1a,996dc51ad14d3a0f
Normal-3038.png,Normal,Normal/images/Normal-3038.png,Normal/masks/Normal-3038.png,True,G11873,test,3aab7fba78b174566fa867f33c00c9a6,dc28539b4dc71ed0
Normal-3039.png,Normal,Normal/images/Normal-3039.png,Normal/masks/Normal-3039.png,True,G11874,train,04002cb09a910ddaa9cc4921936dc976,dc3d424b1dc6b8b8
Normal-304.png,Normal,Normal/images/Normal-304.png,Normal/masks/Normal-304.png,True,G11875,train,d50ea8da9e4ec98f8ecc4b67c688f298,91fc873a8d2c2dcc
Normal-3040.png,Normal,Normal/images/Normal-3040.png,Normal/masks/Normal-3040.png,True,G11876,train,498b1e3e643850c70cef15fac701cfb6,d56fc25b4d28181f
Normal-3041.png,Normal,Normal/images/Normal-3041.png,Normal/masks/Normal-3041.png,True,G11877,train,876b42016b3305a6342801c5f12d2eff,dd2cc7535d430f0c
Normal-3042.png,Normal,Normal/images/Normal-3042.png,Normal/masks/Normal-3042.png,True,G11878,train,7e8fe150388d5ea76d768feea2bd78f8,dd2d521b4d4a1f4a
Normal-3043.png,Normal,Normal/images/Normal-3043.png,Normal/masks/Normal-3043.png,True,G11879,train,1fd8e047c0eb829c05a4adc2a20d3042,d52f461a486e386f
Normal-3044.png,Normal,Normal/images/Normal-3044.png,Normal/masks/Normal-3044.png,True,G11880,train,24af39aeede581aa8d19fae9ac2c920f,c93dc61e144f1e59
Normal-3045.png,Normal,Normal/images/Normal-3045.png,Normal/masks/Normal-3045.png,True,G11881,train,fcce05dc6da0b6c8293d0ff69bbeae02,dc2fc31d4a2d194e
Normal-3046.png,Normal,Normal/images/Normal-3046.png,Normal/masks/Normal-3046.png,True,G11882,train,9c8bbd747f1add94bb114bd5506f5647,d9595042531b5f4f
Normal-3047.png,Normal,Normal/images/Normal-3047.png,Normal/masks/Normal-3047.png,True,G11883,test,d30e18be4443f3fd82000f4241d454dd,983d061e051f795f
Normal-3048.png,Normal,Normal/images/Normal-3048.png,Normal/masks/Normal-3048.png,True,G11884,test,d6b0862acbe5fbda069716ae2e7af367,c829475f1fc42b9c
Normal-3049.png,Normal,Normal/images/Normal-3049.png,Normal/masks/Normal-3049.png,True,G11885,train,8b30bda50fffdbc5474dae57fdfe94b1,d92c8e03534e7dd2
Normal-305.png,Normal,Normal/images/Normal-305.png,Normal/masks/Normal-305.png,True,G11886,train,af53600f0f663daa47d39c18ce3e8f9e,813bc63a9c623f96
Normal-3050.png,Normal,Normal/images/Normal-3050.png,Normal/masks/Normal-3050.png,True,G11887,train,11085ce90b6f9cdd7e38e51212183364,802c47d33dc77cb8
Normal-3051.png,Normal,Normal/images/Normal-3051.png,Normal/masks/Normal-3051.png,True,G11888,test,81e971d46ab142f0c96dfca800a42fc3,886f061a0f1f1f4f
Normal-3052.png,Normal,Normal/images/Normal-3052.png,Normal/masks/Normal-3052.png,True,G11889,train,26f14fbe97413ef241f36f90f63b1a92,817d861a5fce7868
Normal-3053.png,Normal,Normal/images/Normal-3053.png,Normal/masks/Normal-3053.png,True,G11890,train,8fcd5b2c0e11b02e542998157944e829,dd2cd20b0d4c1fbc
Normal-3054.png,Normal,Normal/images/Normal-3054.png,Normal/masks/Normal-3054.png,True,G11891,train,508dc1d9f0491c4839a0c164a843717f,dd2c430b5c2a3dbc
Normal-3055.png,Normal,Normal/images/Normal-3055.png,Normal/masks/Normal-3055.png,True,G11892,val,8580d0e5abfcbd172bc7eb3ad09ab9e1,d93d46021aed6a3d
Normal-3056.png,Normal,Normal/images/Normal-3056.png,Normal/masks/Normal-3056.png,True,G11893,val,201b284ecc9e3d32d886b1873ef2167c,d945165a297ae1e6
Normal-3057.png,Normal,Normal/images/Normal-3057.png,Normal/masks/Normal-3057.png,True,G11894,val,0b09a6341d720d86218dd3fcd3fff242,c82dd61b1d431b5d
Normal-3058.png,Normal,Normal/images/Normal-3058.png,Normal/masks/Normal-3058.png,True,G11895,train,e27ec2614d9d75da7a5142459fdc0c0d,dd08440a1f4e7e7d
Normal-3059.png,Normal,Normal/images/Normal-3059.png,Normal/masks/Normal-3059.png,True,G11896,train,866329797840f03924e0fd6ad93cf7ee,dd2cd35b4c481e4d
Normal-306.png,Normal,Normal/images/Normal-306.png,Normal/masks/Normal-306.png,True,G11897,train,3b2db87344479b33c8fdec1c2750dbf1,956fab2b890d58c5
Normal-3060.png,Normal,Normal/images/Normal-3060.png,Normal/masks/Normal-3060.png,True,G11898,train,7dbc88b221d8dc4403fbffcf11db0ccc,d16b561a5e4e5b0a
Normal-3061.png,Normal,Normal/images/Normal-3061.png,Normal/masks/Normal-3061.png,True,G11899,train,1af0374da3aa5a85ee08501a5fd54fc6,917e96181f3a1a5e
Normal-3062.png,Normal,Normal/images/Normal-3062.png,Normal/masks/Normal-3062.png,True,G11900,train,4cf8fd15ee7bd85d9d09c8b14f179f1d,d43d43db4b4f0b0a
Normal-3063.png,Normal,Normal/images/Normal-3063.png,Normal/masks/Normal-3063.png,True,G11901,train,3c621f09055deaa4084bbf3ec11f1000,917f07161e0d1f1e
Normal-3064.png,Normal,Normal/images/Normal-3064.png,Normal/masks/Normal-3064.png,True,G11902,train,ee2db802dad915dfa39693f716b4c0c6,917947061a174f5f
Normal-3065.png,Normal,Normal/images/Normal-3065.png,Normal/masks/Normal-3065.png,True,G11903,train,8abe6409a95bdd157a17ce9d5e4873a8,9178c61615575f1e
Normal-3066.png,Normal,Normal/images/Normal-3066.png,Normal/masks/Normal-3066.png,True,G11904,train,5cb1a2424dc144b8658cdf00d36aeaa8,817ac65a51477b55
Normal-3067.png,Normal,Normal/images/Normal-3067.png,Normal/masks/Normal-3067.png,True,G11905,train,aebc7a47844abcd035fc7607749541f9,887f871e1e0f1a1b
Normal-3068.png,Normal,Normal/images/Normal-3068.png,Normal/masks/Normal-3068.png,True,G11906,val,073927810f26442787d6aafe978bab4c,d03f56421b1ded4a
Normal-3069.png,Normal,Normal/images/Normal-3069.png,Normal/masks/Normal-3069.png,True,G11907,train,eac68a5a79a0347692970f553b38f413,c87b5b4c583d343c
Normal-307.png,Normal,Normal/images/Normal-307.png,Normal/masks/Normal-307.png,True,G11908,train,29fe44489d8ed48fc9eaad4e8a2cba72,9165875ac969786d
Normal-3070.png,Normal,Normal/images/Normal-3070.png,Normal/masks/Normal-3070.png,True,G11909,train,f559fd5c5af6e958ff41f928f23ea7fd,dd2ad35b1d481c1d
Normal-3071.png,Normal,Normal/images/Normal-3071.png,Normal/masks/Normal-3071.png,True,G11910,val,93aaed0e3107c561b75fe7662111027f,d93944c61e174e5e
Normal-3072.png,Normal,Normal/images/Normal-3072.png,Normal/masks/Normal-3072.png,True,G11911,test,7a0213b4d00f25d4fc9d7c9e07bf2fdf,dd3dc21b444f1a4d
Normal-3073.png,Normal,Normal/images/Normal-3073.png,Normal/masks/Normal-3073.png,True,G11912,test,515a266a3225e582912e04fde0764595,85781e18195f7a79
Normal-3074.png,Normal,Normal/images/Normal-3074.png,Normal/masks/Normal-3074.png,True,G11913,val,b28e0fc4f9d346252ae6d28962b0fdb0,890f065b4d7871e7
Normal-3075.png,Normal,Normal/images/Normal-3075.png,Normal/masks/Normal-3075.png,True,G11914,train,0e638a4652d528e1930cffa269f3d92d,f96ad21b093b0aa7
Normal-3076.png,Normal,Normal/images/Normal-3076.png,Normal/masks/Normal-3076.png,True,G11915,test,e2eae4641937c3fee9d6cc0a21c5efcc,d93cc61b1c4d39e4
Normal-3077.png,Normal,Normal/images/Normal-3077.png,Normal/masks/Normal-3077.png,True,G11916,train,6057c939b6878470a93039dc8a9461e9,c93d021d5e7e38e8
Normal-3078.png,Normal,Normal/images/Normal-3078.png,Normal/masks/Normal-3078.png,True,G11917,train,5a15ccd36b472014b686191db71cfb3f,893c07185a5f78eb
Normal-3079.png,Normal,Normal/images/Normal-3079.png,Normal/masks/Normal-3079.png,True,G11918,train,dc24c88366125df378b048da80897b4d,887907165fdc78ca
Normal-308.png,Normal,Normal/images/Normal-308.png,Normal/masks/Normal-308.png,True,G11919,train,39e89790dad854adef50d5283fb2b9e0,957c873a8d3d2c8c
Normal-3080.png,Normal,Normal/images/Normal-3080.png,Normal/masks/Normal-3080.png,True,G11920,train,ac14d4bfde462eaa22521410e3a2b7d4,897b165a59e879a1
Normal-3081.png,Normal,Normal/images/Normal-3081.png,Normal/masks/Normal-3081.png,True,G11921,train,c78ea5e199e1af35833c0fed9b3d766c,9d2dc613464f1b0f
Normal-3082.png,Normal,Normal/images/Normal-3082.png,Normal/masks/Normal-3082.png,True,G11922,test,9810309204105d86b257bea810332358,d96b061a5b4f28b9
Normal-3083.png,Normal,Normal/images/Normal-3083.png,Normal/masks/Normal-3083.png,True,G11923,train,b462b2f0e0caca2020adf80773cf87aa,9c6e421b1d6e78f0
Normal-3084.png,Normal,Normal/images/Normal-3084.png,Normal/masks/Normal-3084.png,True,G11924,train,f8aa92914b6a74188e3fafd3223a746d,9971164e1b3bcc6c
Normal-3085.png,Normal,Normal/images/Normal-3085.png,Normal/masks/Normal-3085.png,True,G11925,train,d6aa2ba0e4fd5fa14ff1f20bd55d09ae,dd2e825f53421f1c
Normal-3086.png,Normal,Normal/images/Normal-3086.png,Normal/masks/Normal-3086.png,True,G11926,train,357e4c805f6fea445621d87a99c7675e,d42f425f0f0a2f3c
Normal-3087.png,Normal,Normal/images/Normal-3087.png,Normal/masks/Normal-3087.png,True,G11927,train,149c359ef327716e0679d9e7e37b41d2,cc2cc24b0d6c6b3f
Normal-3088.png,Normal,Normal/images/Normal-3088.png,Normal/masks/Normal-3088.png,True,G11928,train,22b53f78ee2374722f3acae09af4d5e0,d53c4f0e194b6d38
Normal-3089.png,Normal,Normal/images/Normal-3089.png,Normal/masks/Normal-3089.png,True,G11929,train,9a48ed49ca911d175b8f31b00f5f2cd1,8969879e059f3987
Normal-309.png,Normal,Normal/images/Normal-309.png,Normal/masks/Normal-309.png,True,G11930,val,1eb534560570d5c44a19e724e74aea13,8129e25a58597ddd
Normal-3090.png,Normal,Normal/images/Normal-3090.png,Normal/masks/Normal-3090.png,True,G11931,test,6c1dde331fd7fabfb6e3174f818f9167,997b471a0e653a65
Normal-3091.png,Normal,Normal/images/Normal-3091.png,Normal/masks/Normal-3091.png,True,G11932,val,4e7dd26ce09c58d88a850edb56432500,c07fc6d758544c78
Normal-3092.png,Normal,Normal/images/Normal-3092.png,Normal/masks/Normal-3092.png,True,G11933,val,2bf7a893f662e4d68852f0c064cc6ebf,987f065a434f3969
Normal-3093.png,Normal,Normal/images/Normal-3093.png,Normal/masks/Normal-3093.png,True,G11934,test,a970e4cec4048821970536e91c2586d0,817907965a176e1f
Normal-3094.png,Normal,Normal/images/Normal-3094.png,Normal/masks/Normal-3094.png,True,G11935,train,d46a19691d8e111d538089ea616f39a6,8d3ed21b840f994f
Normal-3095.png,Normal,Normal/images/Normal-3095.png,Normal/masks/Normal-3095.png,True,G11936,train,337f8017ca787eb3ba451fb0a71253bd,d979051e164f1e1b
Normal-3096.png,Normal,Normal/images/Normal-3096.png,Normal/masks/Normal-3096.png,True,G11937,train,c5982fdafb66e88e1fc6232a3674c3e1,dd3d825b144b194f
Normal-3097.png,Normal,Normal/images/Normal-3097.png,Normal/masks/Normal-3097.png,True,G11938,train,7c6b64b78775e54154018bdfdeb8530e,d97dc61f1f441e10
Normal-3098.png,Normal,Normal/images/Normal-3098.png,Normal/masks/Normal-3098.png,True,G11939,train,11c59f8e5c163ea6f50fee34d066e3a8,9979c55a1d4d1a1d
Normal-3099.png,Normal,Normal/images/Normal-3099.png,Normal/masks/Normal-3099.png,True,G11940,train,714a977cbefdf3b10b0b17de70757b4f,dd2fc24b1d420f3c
Normal-31.png,Normal,Normal/images/Normal-31.png,Normal/masks/Normal-31.png,True,G11941,train,24251ae0f604f069e10265f46c6425ca,dc3cc25f51425f58
Normal-310.png,Normal,Normal/images/Normal-310.png,Normal/masks/Normal-310.png,True,G11942,val,49c4b349126155ba43de195b16085309,d46ce72be92d2c0c
Normal-3100.png,Normal,Normal/images/Normal-3100.png,Normal/masks/Normal-3100.png,True,G11943,train,25d269c46feabcee53a3b772750490fe,993e461b58e9e4b8
Normal-3101.png,Normal,Normal/images/Normal-3101.png,Normal/masks/Normal-3101.png,True,G11944,train,ebd0775d81d17337b1fe8af212f013d1,dd74420b0b0f3d3d
Normal-3102.png,Normal,Normal/images/Normal-3102.png,Normal/masks/Normal-3102.png,True,G11945,test,63e29d0735fa9ee0014fde23e342f761,d939c2571a454f1d
Normal-3103.png,Normal,Normal/images/Normal-3103.png,Normal/masks/Normal-3103.png,True,G11946,train,e750b20356097556f5ab52bb3046d3ab,887d861edd471e1c
Normal-3104.png,Normal,Normal/images/Normal-3104.png,Normal/masks/Normal-3104.png,True,G11947,test,bcc6d97184d5452f86457082550a3c85,9979871a194f5a99
Normal-3105.png,Normal,Normal/images/Normal-3105.png,Normal/masks/Normal-3105.png,True,G11948,train,586ca87fda5d1ee0a25369239ba2d5c9,d179465e184e6d39
Normal-3106.png,Normal,Normal/images/Normal-3106.png,Normal/masks/Normal-3106.png,True,G11949,train,f5c3633c1f69c184636b7057f4ba3924,992b861e0e5e7a59
Normal-3107.png,Normal,Normal/images/Normal-3107.png,Normal/masks/Normal-3107.png,True,G11950,test,519a15dd911c5fc20829b3f360c9735b,8909165e5a7b7b68
Normal-3108.png,Normal,Normal/images/Normal-3108.png,Normal/masks/Normal-3108.png,True,G11951,train,3adcab7a46bce2f20075226d27f2ba60,f879c21b1f461b19
Normal-3109.png,Normal,Normal/images/Normal-3109.png,Normal/masks/Normal-3109.png,True,G11952,train,dafe80d7369086912a393201c49008a8,dc3d424b0b426f1f
Normal-311.png,Normal,Normal/images/Normal-311.png,Normal/masks/Normal-311.png,True,G11953,val,c29617b92116af444365f9e8c7710d5b,913dc61bdc6938c6
Normal-3110.png,Normal,Normal/images/Normal-3110.png,Normal/masks/Normal-3110.png,True,G11954,train,6afa3ef675512b8bdc42389245b148f8,d53bc2161b0b5f0d
Normal-3111.png,Normal,Normal/images/Normal-3111.png,Normal/masks/Normal-3111.png,True,G11955,val,f49939ab8ceac624fc656c0bc7a03c2c,8828571f1e4f19cf
Normal-3112.png,Normal,Normal/images/Normal-3112.png,Normal/masks/Normal-3112.png,True,G11956,train,bd9cd48c751785615db2d67d8fc8d9e2,817f041e1a1f5f1b
Normal-3113.png,Normal,Normal/images/Normal-3113.png,Normal/masks/Normal-3113.png,True,G11957,train,1b093447e28d9ed92d88df636c5c1be3,8978875a194d7a79
Normal-3114.png,Normal,Normal/images/Normal-3114.png,Normal/masks/Normal-3114.png,True,G11958,train,bc0a3bb6a6d586851a7319a41d447877,997bc71e5a451a19
Normal-3115.png,Normal,Normal/images/Normal-3115.png,Normal/masks/Normal-3115.png,True,G11959,val,b21fb744fc1ac22a56f1d2850b19b7c7,817901d7961afb47
Normal-3116.png,Normal,Normal/images/Normal-3116.png,Normal/masks/Normal-3116.png,True,G11960,train,d81ebfaf30457ae87040bdc8641beadc,cd7cd2434d0c0f2f
Normal-3117.png,Normal,Normal/images/Normal-3117.png,Normal/masks/Normal-3117.png,True,G11961,train,8deb1bd9671d12d2225f4902bc0c9b17,d43c43c31b0f6cec
Normal-3118.png,Normal,Normal/images/Normal-3118.png,Normal/masks/Normal-3118.png,True,G11962,train,63e585db86f996f269948f1e70d5b766,dc7a430f0d292dbc
Normal-3119.png,Normal,Normal/images/Normal-3119.png,Normal/masks/Normal-3119.png,True,G11963,train,6ae0cea582d050d8096d1400aa53a43b,81798616de5e7a0b
Normal-312.png,Normal,Normal/images/Normal-312.png,Normal/masks/Normal-312.png,True,G11964,train,f2edd99f7f48625d4311cc6513227bc4,c11ec71ab9e132e6
Normal-3120.png,Normal,Normal/images/Normal-3120.png,Normal/masks/Normal-3120.png,True,G11965,train,37098ca812d0cd018ddae24ea3dd43a6,d92dc69e144b1d1d
Normal-3121.png,Normal,Normal/images/Normal-3121.png,Normal/masks/Normal-3121.png,True,G11966,train,17039f9521fe6df6732e63d4068b839a,81790e56195fea5a
Normal-3122.png,Normal,Normal/images/Normal-3122.png,Normal/masks/Normal-3122.png,True,G11967,train,2c1c7f502f3fde43b32443f3d9b2f70c,9d29c61e49691ce7
Normal-3123.png,Normal,Normal/images/Normal-3123.png,Normal/masks/Normal-3123.png,True,G11968,train,dc73a59b2d98e5d8b5fce93c974b4571,9938461f9f6a38e8
Normal-3124.png,Normal,Normal/images/Normal-3124.png,Normal/masks/Normal-3124.png,True,G11969,test,1d77740c536eadd532cb2644866948fb,d97a061b1d6d38f0
Normal-3125.png,Normal,Normal/images/Normal-3125.png,Normal/masks/Normal-3125.png,True,G11970,val,27f877065873a39f7d694ac8071e5416,886b479e1b456f19
Normal-3126.png,Normal,Normal/images/Normal-3126.png,Normal/masks/Normal-3126.png,True,G11971,test,9e0da15ef8dba52d230acfaeee32d665,d06d421d4f2d38b7
Normal-3127.png,Normal,Normal/images/Normal-3127.png,Normal/masks/Normal-3127.png,True,G11972,train,d57ff3e79ab4f7773060f29b4af04457,85691e5d59596959
Normal-3128.png,Normal,Normal/images/Normal-3128.png,Normal/masks/Normal-3128.png,True,G11973,val,b2ec8d1040f90181a3c08661135c5296,9169061e1bcd7e78
Normal-3129.png,Normal,Normal/images/Normal-3129.png,Normal/masks/Normal-3129.png,True,G11974,test,395d291f75f9e43d97f83406a6d6bc04,dd29c21b1d421bbd
Normal-313.png,Normal,Normal/images/Normal-313.png,Normal/masks/Normal-313.png,True,G11975,train,cf4a0e504ade67e023786007e0633359,917ed65e5c091d86
Normal-3130.png,Normal,Normal/images/Normal-3130.png,Normal/masks/Normal-3130.png,True,G11976,train,1f804c2ab1a39e93acafad3dfbf1d239,80bfc61e1a55393d
Normal-3131.png,Normal,Normal/images/Normal-3131.png,Normal/masks/Normal-3131.png,True,G11977,train,6aa6a457d2cfd79cb7147af08bebeea3,992f071e15175e55
Normal-3132.png,Normal,Normal/images/Normal-3132.png,Normal/masks/Normal-3132.png,True,G11978,train,dfd758a162fbae1a913515190c44a464,8c4c42535f4d4d3f
Normal-3133.png,Normal,Normal/images/Normal-3133.png,Normal/masks/Normal-3133.png,True,G11979,train,c4e223e20ba05c88f18f6d465d00d0c8,b909061e5b1d5c5f
Normal-3134.png,Normal,Normal/images/Normal-3134.png,Normal/masks/Normal-3134.png,True,G11980,train,06d0528b29fb39fbb9f85aa1af2d3195,91691e1e1b5e7839
Normal-3135.png,Normal,Normal/images/Normal-3135.png,Normal/masks/Normal-3135.png,True,G11981,train,5a53c7622aba6dba0a4664b4e3d76ec5,813b061e1c5f7f19
Normal-3136.png,Normal,Normal/images/Normal-3136.png,Normal/masks/Normal-3136.png,True,G11982,train,6acb06377c6e4e8b5fec33eba236bf05,8168071a5f5b795d
Normal-3137.png,Normal,Normal/images/Normal-3137.png,Normal/masks/Normal-3137.png,True,G11983,train,f5172464077b7e3c04db15c60804db16,997b061a591f195d
Normal-3138.png,Normal,Normal/images/Normal-3138.png,Normal/masks/Normal-3138.png,True,G11984,test,4c7d606745cc115f3f7ad0ee60df4837,c97d061a1dc96b3c
Normal-3139.png,Normal,Normal/images/Normal-3139.png,Normal/masks/Normal-3139.png,True,G11985,val,7cc44a3b6ab9daa7c2de653d7db552f5,cc3e434b146c3eb3
Normal-314.png,Normal,Normal/images/Normal-314.png,Normal/masks/Normal-314.png,True,G11986,train,9466dc7d49a68a22e17bc6b218329d82,9969c65acb5b5845
Normal-3140.png,Normal,Normal/images/Normal-3140.png,Normal/masks/Normal-3140.png,True,G11987,train,f9c965ebb665f0b4fa809976a8c87929,c93dc61a1d4a39ad
Normal-3141.png,Normal,Normal/images/Normal-3141.png,Normal/masks/Normal-3141.png,True,G11988,train,f6a3b4f27f9a369d1cbd51be9542a5ed,992dc70a1d455f1d
Normal-3142.png,Normal,Normal/images/Normal-3142.png,Normal/masks/Normal-3142.png,True,G11989,train,0369dd7e4d9364934137d03b628383ba,996d86128f0f1a9f
Normal-3143.png,Normal,Normal/images/Normal-3143.png,Normal/masks/Normal-3143.png,True,G11990,train,0935d905258cd9887fb3e7693945454d,81790606195f7f79
Normal-3144.png,Normal,Normal/images/Normal-3144.png,Normal/masks/Normal-3144.png,True,G11991,val,e21aa358e4f93c9f5f65eaad3ae17cc5,9979871b1e061e1f
Normal-3145.png,Normal,Normal/images/Normal-3145.png,Normal/masks/Normal-3145.png,True,G11992,val,a3a8a7e4d5dc2ec8992acab94ca345e8,8979d656191d4b1b
Normal-3146.png,Normal,Normal/images/Normal-3146.png,Normal/masks/Normal-3146.png,True,G11993,test,33854e0bb32fabe9f36eebdc65ff546b,d52b471b0b4f1a1b
Normal-3147.png,Normal,Normal/images/Normal-3147.png,Normal/masks/Normal-3147.png,True,G11994,test,0d81169f64a8f2ee8754161bc079c8e8,997dc6521a494f1e
Normal-3148.png,Normal,Normal/images/Normal-3148.png,Normal/masks/Normal-3148.png,True,G11995,train,e6aef47399deca29afded4fb05672a08,d939c6061d565f1a
Normal-3149.png,Normal,Normal/images/Normal-3149.png,Normal/masks/Normal-3149.png,True,G11996,train,6b5e9755e371581049553d9957e23311,c938060e1ee97cf9
Normal-315.png,Normal,Normal/images/Normal-315.png,Normal/masks/Normal-315.png,True,G11997,val,7d037177bda90c17d9b30dd1db58b48d,903de73aa96d3861
Normal-3150.png,Normal,Normal/images/Normal-3150.png,Normal/masks/Normal-3150.png,True,G11998,test,b3971201f012ec9e7b9843b4afc9c3c9,dd6cd0925f4b061e
Normal-3151.png,Normal,Normal/images/Normal-3151.png,Normal/masks/Normal-3151.png,True,G11999,train,a79b94f870ae82b0b6049124ddad61e4,897b041e5f0b5b39
Normal-3152.png,Normal,Normal/images/Normal-3152.png,Normal/masks/Normal-3152.png,True,G12000,test,6979809eb49edf60c84ca8c04156e339,9969061e5f5e185b
Normal-3153.png,Normal,Normal/images/Normal-3153.png,Normal/masks/Normal-3153.png,True,G12001,test,3a45bd8ba7c19a0fb641d70dd48042f2,dcae925bc24b2b43
Normal-3154.png,Normal,Normal/images/Normal-3154.png,Normal/masks/Normal-3154.png,True,G12002,train,dae324fb54cf536564ac3bc0af886b03,dd2c42430b4caf3f
Normal-3155.png,Normal,Normal/images/Normal-3155.png,Normal/masks/Normal-3155.png,True,G12003,train,813f94a14e3c8ba3253d43c47e5e2543,d979161e19093bb9
Normal-3156.png,Normal,Normal/images/Normal-3156.png,Normal/masks/Normal-3156.png,True,G12004,train,6c05ca7e8d85a1e4c4d53c0fca8705d7,d9abd21948597f48
Normal-3157.png,Normal,Normal/images/Normal-3157.png,Normal/masks/Normal-3157.png,True,G12005,train,892cd52f53926f7f8d59cac5ef37010c,c03c02cb3ecb3ddc
Normal-3158.png,Normal,Normal/images/Normal-3158.png,Normal/masks/Normal-3158.png,True,G12006,train,8384fe274133a25a1ce360804bf681b4,dd3d421d0a2c3db5
Normal-3159.png,Normal,Normal/images/Normal-3159.png,Normal/masks/Normal-3159.png,True,G12007,train,bb7724f738e70ca32af58e53cbf309c3,913d8614191ffc3d
Normal-316.png,Normal,Normal/images/Normal-316.png,Normal/masks/Normal-316.png,True,G12008,train,71a46f489fe84c7e823cf82707ded17d,d939c61e584d1dc5
Normal-3160.png,Normal,Normal/images/Normal-3160.png,Normal/masks/Normal-3160.png,True,G12009,train,c4219467246e31fc8cc6928c3ea6380e,983b95121e5d7e58
Normal-3161.png,Normal,Normal/images/Normal-3161.png,Normal/masks/Normal-3161.png,True,G12010,train,f076bb2e5895122e226e0640ee3674e4,993dc65a19c54f19
Normal-3162.png,Normal,Normal/images/Normal-3162.png,Normal/masks/Normal-3162.png,True,G12011,train,7536072b4807877f7fd50d4e5a938884,817907079a9e574f
Normal-3163.png,Normal,Normal/images/Normal-3163.png,Normal/masks/Normal-3163.png,True,G12012,train,0e1a3d669131576e5a46c98a4fe57fb2,d93bc21b0d0f3b19
Normal-3164.png,Normal,Normal/images/Normal-3164.png,Normal/masks/Normal-3164.png,True,G12013,val,aeba204de73dfcbdaa9fe8799b6e3c7b,c83dc7431a4aaf1b
Normal-3165.png,Normal,Normal/images/Normal-3165.png,Normal/masks/Normal-3165.png,True,G12014,val,b225cbe06d4ee1721b0310d719cc49b5,817a47861b5d4e5b
Normal-3166.png,Normal,Normal/images/Normal-3166.png,Normal/masks/Normal-3166.png,True,G12015,train,6b53c709365f77b6a1c8c5dd27de3480,8179861a785f29bb
Normal-3167.png,Normal,Normal/images/Normal-3167.png,Normal/masks/Normal-3167.png,True,G12016,train,601b305397907d5463b1fa459cfb767d,817b56065a5f5b19
Normal-3168.png,Normal,Normal/images/Normal-3168.png,Normal/masks/Normal-3168.png,True,G12017,train,c877f99833661095e973e89651357309,bc2d8712835f154f
Normal-3169.png,Normal,Normal/images/Normal-3169.png,Normal/masks/Normal-3169.png,True,G12018,train,dfa8c096d000719fa9fe9a3a6b953e91,987fc7121ec71a1a
Normal-317.png,Normal,Normal/images/Normal-317.png,Normal/masks/Normal-317.png,True,G12019,train,de4d558c6612d3a468df2c6dc96af016,912bc25ad8795979
Normal-3170.png,Normal,Normal/images/Normal-3170.png,Normal/masks/Normal-3170.png,True,G12020,train,726f46305831919252079647853fac6e,d87905865e9147d7
Normal-3171.png,Normal,Normal/images/Normal-3171.png,Normal/masks/Normal-3171.png,True,G12021,train,29d186a55ef16a362432564f9a365e3a,c43d02c24b1f5f6b
Normal-3172.png,Normal,Normal/images/Normal-3172.png,Normal/masks/Normal-3172.png,True,G12022,val,19a2428157d27a5cab29f2028d9e001b,89395e4f114d1d79
Normal-3173.png,Normal,Normal/images/Normal-3173.png,Normal/masks/Normal-3173.png,True,G12023,train,784cf8a304946deccb592628e14b90db,c939c65ed4465a53
Normal-3174.png,Normal,Normal/images/Normal-3174.png,Normal/masks/Normal-3174.png,True,G12024,train,5dee0dcdb1968bf4e0e82549a64d3db2,c0a86ad31b9f2e3a
Normal-3175.png,Normal,Normal/images/Normal-3175.png,Normal/masks/Normal-3175.png,True,G12025,val,cab4d446ab0cebf98249b87b370a38e0,c92b17875b0cce1b
Normal-3176.png,Normal,Normal/images/Normal-3176.png,Normal/masks/Normal-3176.png,True,G12026,train,4bb2b9ba1b0509d6194bdc14806200cb,f979a58714c1c715
Normal-3177.png,Normal,Normal/images/Normal-3177.png,Normal/masks/Normal-3177.png,True,G12027,test,a4e90e749de0f5b61df02578522ddbcb,813fc6861a19ee6e
Normal-3178.png,Normal,Normal/images/Normal-3178.png,Normal/masks/Normal-3178.png,True,G12028,test,162b76d97ced4399602809ea7d0f9441,9968065b597d39a9
Normal-3179.png,Normal,Normal/images/Normal-3179.png,Normal/masks/Normal-3179.png,True,G12029,train,64ae733765748e12739281084377b741,d979061e18697bb8
Normal-318.png,Normal,Normal/images/Normal-318.png,Normal/masks/Normal-318.png,True,G12030,test,ed7f8dd06e13c0312c200b369474aaee,d138e21afe4918dd
Normal-3180.png,Normal,Normal/images/Normal-3180.png,Normal/masks/Normal-3180.png,True,G12031,train,45c7e4b7ee54216ddf292b982c05209c,d93cc71b19854e39
Normal-3181.png,Normal,Normal/images/Normal-3181.png,Normal/masks/Normal-3181.png,True,G12032,train,4dca92bb81d11543dea8e8152511a9b1,c82b55465e5d4d4b
Normal-3182.png,Normal,Normal/images/Normal-3182.png,Normal/masks/Normal-3182.png,True,G12033,train,83853ea12f43886431a1832186e200fc,913d46965a03ef1e
Normal-3183.png,Normal,Normal/images/Normal-3183.png,Normal/masks/Normal-3183.png,True,G12034,val,a74b555d5861f31065c42d3f4d0868b8,807847961e575f1e
Normal-3184.png,Normal,Normal/images/Normal-3184.png,Normal/masks/Normal-3184.png,True,G12035,train,337acb7be9a3cabae0e90ae172e612cc,d93d821b1d7ab8c8
Normal-3185.png,Normal,Normal/images/Normal-3185.png,Normal/masks/Normal-3185.png,True,G12036,train,d3a86d08d68e98358ad25eb091954c84,992dc61a1c4f397c
Normal-3186.png,Normal,Normal/images/Normal-3186.png,Normal/masks/Normal-3186.png,True,G12037,train,af8bd4c6bf8c3e16bdc91e7f9740f5ee,9d2c461b7d6c3ce0
Normal-3187.png,Normal,Normal/images/Normal-3187.png,Normal/masks/Normal-3187.png,True,G12038,train,63f4c86d9bbec95c1eb7e42e3c2ed9fb,dc3d430b0f0f2d2e
Normal-3188.png,Normal,Normal/images/Normal-3188.png,Normal/masks/Normal-3188.png,True,G12039,train,3c0a44f8a1146e63acd8183da13578be,8979d6d6581cc25b
Normal-3189.png,Normal,Normal/images/Normal-3189.png,Normal/masks/Normal-3189.png,True,G12040,train,1099b7ff3322ff001d59b34bf07cf6b7,992e06185f7d78e1
Normal-319.png,Normal,Normal/images/Normal-319.png,Normal/masks/Normal-319.png,True,G12041,train,c655a9ce187aa2e525bf769389f723d3,916a871acb5d58cd
Normal-3190.png,Normal,Normal/images/Normal-3190.png,Normal/masks/Normal-3190.png,True,G12042,train,2ab9e12f118437fa2ac291d8cd4bde85,d82d86971a594d4f
Normal-3191.png,Normal,Normal/images/Normal-3191.png,Normal/masks/Normal-3191.png,True,G12043,train,9fc929d6fe3a2e099bd5347ea33c807a,d42853434f5c4e5f
Normal-3192.png,Normal,Normal/images/Normal-3192.png,Normal/masks/Normal-3192.png,True,G12044,train,543f636bd86df4b3586228d8206e3aba,817b56191b293ba7
Normal-3193.png,Normal,Normal/images/Normal-3193.png,Normal/masks/Normal-3193.png,True,G12045,train,91b3a0017a521d7392a7dec6adcabec8,9979065e5869ed38
Normal-3194.png,Normal,Normal/images/Normal-3194.png,Normal/masks/Normal-3194.png,True,G12046,train,032fa522c84967834e8de640c8fb43f1,dd2a421dc20bde4f
Normal-3195.png,Normal,Normal/images/Normal-3195.png,Normal/masks/Normal-3195.png,True,G12047,train,cb48d983df87f50547a53a34332c9dc9,dcbd4243570bca4b
Normal-3196.png,Normal,Normal/images/Normal-3196.png,Normal/masks/Normal-3196.png,True,G12048,train,674d8740bbc99129a7e7ab7b310b78d6,d568470b196f2a3e
Normal-3197.png,Normal,Normal/images/Normal-3197.png,Normal/masks/Normal-3197.png,True,G12049,val,e386f4e2368140e6c4e4b829dc1570b5,993f861a1fac38a9
Normal-3198.png,Normal,Normal/images/Normal-3198.png,Normal/masks/Normal-3198.png,True,G12050,val,ddc281acff6e9b6886d41b2849159590,dd3cc21b494b1b4d
Normal-3199.png,Normal,Normal/images/Normal-3199.png,Normal/masks/Normal-3199.png,True,G12051,val,8e074a91a4fa600267bb05f17d28028d,91784746164b5f3d
Normal-32.png,Normal,Normal/images/Normal-32.png,Normal/masks/Normal-32.png,True,G12052,test,28ff475313953f426c43ed36bc2550c6,91691e961e5adc1e
Normal-320.png,Normal,Normal/images/Normal-320.png,Normal/masks/Normal-320.png,True,G12053,test,159eabf48d617b615c363de645a07c70,9169c65adc4b3969
Normal-3200.png,Normal,Normal/images/Normal-3200.png,Normal/masks/Normal-3200.png,True,G12054,test,0cc91421dd91a24db296f319eba6d726,d17dc6431b052b3b
Normal-3201.png,Normal,Normal/images/Normal-3201.png,Normal/masks/Normal-3201.png,True,G12055,train,382a25f8e758849385096e7ff55993a5,896e16181e591f7b
Normal-3202.png,Normal,Normal/images/Normal-3202.png,Normal/masks/Normal-3202.png,True,G12056,train,10212a7369fae506cc2d0b6ffb9ec582,887f061a151b797e
Normal-3203.png,Normal,Normal/images/Normal-3203.png,Normal/masks/Normal-3203.png,True,G12057,val,e05aedeb52493f3116e495adc738e046,d07f834c284aaf7c
Normal-3204.png,Normal,Normal/images/Normal-3204.png,Normal/masks/Normal-3204.png,True,G12058,train,a345730e82f7cc034516b3925e3d9fa9,813dd7c21b190d6f
Normal-3205.png,Normal,Normal/images/Normal-3205.png,Normal/masks/Normal-3205.png,True,G12059,train,d3655fbf57c369a24bff2a863d7a0c21,9d7cc0139b4d0f1d
Normal-3206.png,Normal,Normal/images/Normal-3206.png,Normal/masks/Normal-3206.png,True,G12060,train,131076e4a4bb0274400cd219e9c2c736,d47c87471d540e3d
Normal-3207.png,Normal,Normal/images/Normal-3207.png,Normal/masks/Normal-3207.png,True,G12061,train,3371bc842a29403e3d87f86be90cda0a,d938c6421f1f4b1d
Normal-3208.png,Normal,Normal/images/Normal-3208.png,Normal/masks/Normal-3208.png,True,G12062,train,4d52ebddbb1d4d02c779918010a1f79a,983f861e88999f87
Normal-3209.png,Normal,Normal/images/Normal-3209.png,Normal/masks/Normal-3209.png,True,G12063,val,ed7ca3687f1119e2d680d7ac3dbd51df,986f865a455b194f
Normal-321.png,Normal,Normal/images/Normal-321.png,Normal/masks/Normal-321.png,True,G12064,test,5dfd69f799e3f9c3edbd993df71ec503,c16f871a89e16b93
Normal-3210.png,Normal,Normal/images/Normal-3210.png,Normal/masks/Normal-3210.png,True,G12065,val,31ffe86b751919da8bcf51c536c3b96a,dc2d430f4f0f0e0f
Normal-3211.png,Normal,Normal/images/Normal-3211.png,Normal/masks/Normal-3211.png,True,G12066,train,3c458a5bab66e5f6792f048950ecb07f,897b85865a1c5d5b
Normal-3212.png,Normal,Normal/images/Normal-3212.png,Normal/masks/Normal-3212.png,True,G12067,train,977ec85365024b83a45b33c8118ef901,897b869e964b1c5a
Normal-3213.png,Normal,Normal/images/Normal-3213.png,Normal/masks/Normal-3213.png,True,G12068,train,a1cb6417f3270f78284cd4a14f516452,8178961c1f5e785b
Normal-3214.png,Normal,Normal/images/Normal-3214.png,Normal/masks/Normal-3214.png,True,G12069,train,c743edeeb67cfd02ca5b567a19a2538f,987bc71e194a58e9
Normal-3215.png,Normal,Normal/images/Normal-3215.png,Normal/masks/Normal-3215.png,True,G12070,train,587a39a28840c4a78c99670437db3cad,d42c52531b4f4f1e
Normal-3216.png,Normal,Normal/images/Normal-3216.png,Normal/masks/Normal-3216.png,True,G12071,train,871d93e853cf7a0ec674eca19be1255b,95680617db5e28fa
Normal-3217.png,Normal,Normal/images/Normal-3217.png,Normal/masks/Normal-3217.png,True,G12072,train,604b877f1db7e66162475b81a3a02d6b,d969071e0f1e5a99
Normal-3218.png,Normal,Normal/images/Normal-3218.png,Normal/masks/Normal-3218.png,True,G12073,val,4e0dacd09b50d09dadedcb68008f2275,d83d4247d3474b1b
Normal-3219.png,Normal,Normal/images/Normal-3219.png,Normal/masks/Normal-3219.png,True,G12074,test,90952c93cd3a886487b014262bfeddcb,8979055e1a1d6b79
Normal-322.png,Normal,Normal/images/Normal-322.png,Normal/masks/Normal-322.png,True,G12075,test,782941d7356c66acbcaf7c7ecbfc1601,d029c23bf8693d69
Normal-3220.png,Normal,Normal/images/Normal-3220.png,Normal/masks/Normal-3220.png,True,G12076,test,aeb979bbfbaf358781fec7df263ec124,817907161e5f5a5e
Normal-3221.png,Normal,Normal/images/Normal-3221.png,Normal/masks/Normal-3221.png,True,G12077,train,8df649d269057f5c55c40e518196f822,997f861a491f186d
Normal-3222.png,Normal,Normal/images/Normal-3222.png,Normal/masks/Normal-3222.png,True,G12078,test,fdffe2482cef0e0b689cfb39a4ae30db,897e451a1bcd3da8
Normal-3223.png,Normal,Normal/images/Normal-3223.png,Normal/masks/Normal-3223.png,True,G12079,train,a9f3cb4dee0109ac450e205054eb19c8,9979c61a154d3ae9
Normal-3224.png,Normal,Normal/images/Normal-3224.png,Normal/masks/Normal-3224.png,True,G12080,train,294458e7d5cd3be9a55f8e8986d32749,b95d421e1d491b1f
Normal-3225.png,Normal,Normal/images/Normal-3225.png,Normal/masks/Normal-3225.png,True,G12081,train,59bce67418fd8a4cb028f2a9bac98c66,996f861a591d784d
Normal-3226.png,Normal,Normal/images/Normal-3226.png,Normal/masks/Normal-3226.png,True,G12082,train,fc3d5a972157b46967cad06bd6758477,dd2ad31b0f461c5c
Normal-3227.png,Normal,Normal/images/Normal-3227.png,Normal/masks/Normal-3227.png,True,G12083,train,01f28da1e78db722f5d1ee8c87f9bc68,d93b04c61a1d4e7b
Normal-3228.png,Normal,Normal/images/Normal-3228.png,Normal/masks/Normal-3228.png,True,G12084,train,cec40113092971552b1aa8910ee86b71,81685e1e191ee9f9
Normal-3229.png,Normal,Normal/images/Normal-3229.png,Normal/masks/Normal-3229.png,True,G12085,train,4f6375aff804ed5afdd55797af620348,9968c6121f6b396d
Normal-323.png,Normal,Normal/images/Normal-323.png,Normal/masks/Normal-323.png,True,G12086,train,b480236d467d3b4887fac1dd03cfb1fb,812bce5acc4b5dc9
Normal-3230.png,Normal,Normal/images/Normal-3230.png,Normal/masks/Normal-3230.png,True,G12087,train,973504d0baae4d44751df4e81075c9f5,cc2d425d0f2a6dbc
Normal-3231.png,Normal,Normal/images/Normal-3231.png,Normal/masks/Normal-3231.png,True,G12088,train,1f21924165dcba83f9b1ec7783329f6c,9169861a591f7979
Normal-3232.png,Normal,Normal/images/Normal-3232.png,Normal/masks/Normal-3232.png,True,G12089,train,9ea4de25976134a8663c4ba9f6aaa791,d17946161a1f4f39
Normal-3233.png,Normal,Normal/images/Normal-3233.png,Normal/masks/Normal-3233.png,True,G12090,train,82079461dee3996451b8570246dce0f9,d93d061f184f3939
Normal-3234.png,Normal,Normal/images/Normal-3234.png,Normal/masks/Normal-3234.png,True,G12091,train,58f49f0432283b181153fed4a440413b,993d861a094f3aed
Normal-3235.png,Normal,Normal/images/Normal-3235.png,Normal/masks/Normal-3235.png,True,G12092,val,0bc5bacfe05135a6203e2a8cb76a008f,d93bc60f96471e50
Normal-3236.png,Normal,Normal/images/Normal-3236.png,Normal/masks/Normal-3236.png,True,G12093,test,430c95656fbd1e4656475c770c6bbd13,c939c616164e1e5f
Normal-3237.png,Normal,Normal/images/Normal-3237.png,Normal/masks/Normal-3237.png,True,G12094,val,8e3d16da267e8adf0387eb5ee22ea4a6,887987065a1f7f58
Normal-3238.png,Normal,Normal/images/Normal-3238.png,Normal/masks/Normal-3238.png,True,G12095,train,c0ab031df10a88703051229ce3b4b2de,dd0c40431f4d7df4
Normal-3239.png,Normal,Normal/images/Normal-3239.png,Normal/masks/Normal-3239.png,True,G12096,train,c411211959f85c9627557a90eaedfdf5,8c3d030b4f28ecfd
Normal-324.png,Normal,Normal/images/Normal-324.png,Normal/masks/Normal-324.png,True,G12097,train,35e32ba7ffe45704eb2470262cadff7e,94be971d0a0e5cd6
Normal-3240.png,Normal,Normal/images/Normal-3240.png,Normal/masks/Normal-3240.png,True,G12098,train,5d2c7e1fb6938358988d595afb9d0a72,d43f424f1b0ced38
Normal-3241.png,Normal,Normal/images/Normal-3241.png,Normal/masks/Normal-3241.png,True,G12099,train,f6f44545c4d49385554c12f50c201017,d5bd06821adb0fca
Normal-3242.png,Normal,Normal/images/Normal-3242.png,Normal/masks/Normal-3242.png,True,G12100,train,537a47e7e05d5b8e55e02fa4ffe9971f,897986961a4f7959
Normal-3243.png,Normal,Normal/images/Normal-3243.png,Normal/masks/Normal-3243.png,True,G12101,train,5d5a0ec848b56a0da29eec2e2b15cb02,897f861e184d6d39
Normal-3244.png,Normal,Normal/images/Normal-3244.png,Normal/masks/Normal-3244.png,True,G12102,train,df62f8e8ac6041c861d4279d203b02e6,dd08831e08edb9f5
Normal-3245.png,Normal,Normal/images/Normal-3245.png,Normal/masks/Normal-3245.png,True,G12103,train,92f3df951ebc00734e99b4e4bc4ae8ea,bc0c435b4d4d5c6e
Normal-3246.png,Normal,Normal/images/Normal-3246.png,Normal/masks/Normal-3246.png,True,G12104,train,eb88b1564be5827d76b9d00e28c290c8,987b07161e4b3b69
Normal-3247.png,Normal,Normal/images/Normal-3247.png,Normal/masks/Normal-3247.png,True,G12105,train,7ee2cc7e3d1f3ef513e6077dee59e42e,cd6b474949c6c2f8
Normal-3248.png,Normal,Normal/images/Normal-3248.png,Normal/masks/Normal-3248.png,True,G12106,train,c333f6dfd009563eddb37eb33dd6b9fb,b93dc6435b454f18
Normal-3249.png,Normal,Normal/images/Normal-3249.png,Normal/masks/Normal-3249.png,True,G12107,train,73691e9d4d0d431e8387c0b3a80f9861,91681e175b5e6939
Normal-325.png,Normal,Normal/images/Normal-325.png,Normal/masks/Normal-325.png,True,G12108,val,fa1e4a449d25c9e4bc9a2f2bc0283288,992dc61aeb6b3868
Normal-3250.png,Normal,Normal/images/Normal-3250.png,Normal/masks/Normal-3250.png,True,G12109,val,5279a6d21769cb60ebf1e69c91fa8d45,d97bd21a485b194b
Normal-3251.png,Normal,Normal/images/Normal-3251.png,Normal/masks/Normal-3251.png,True,G12110,train,69efdf4fbf2a239c1506c041340423d9,f97902d61a190d5f
Normal-3252.png,Normal,Normal/images/Normal-3252.png,Normal/masks/Normal-3252.png,True,G12111,test,763e9a4babdfa4966ac712a32e84952b,c83d43160fedbca8
Normal-3253.png,Normal,Normal/images/Normal-3253.png,Normal/masks/Normal-3253.png,True,G12112,test,aa7bd02a04f0d41cb9dbcf144899c7ce,d43f43514f0e1c4f
Normal-3254.png,Normal,Normal/images/Normal-3254.png,Normal/masks/Normal-3254.png,True,G12113,val,468d691a3693462d4bfbd1473624e657,8178065a3f6f39c5
Normal-3255.png,Normal,Normal/images/Normal-3255.png,Normal/masks/Normal-3255.png,True,G12114,train,8b5516cf57683e07533fd1353eb55896,dd4c921b450a3fe6
Normal-3256.png,Normal,Normal/images/Normal-3256.png,Normal/masks/Normal-3256.png,True,G12115,test,6603a1eef5adb854ec5bebaa65b432d5,d12c06de58ecec71
Normal-3257.png,Normal,Normal/images/Normal-3257.png,Normal/masks/Normal-3257.png,True,G12116,val,bd8d521bf82ad3ecf19314653a8c1856,9939c6161b4b4f78
Normal-3258.png,Normal,Normal/images/Normal-3258.png,Normal/masks/Normal-3258.png,True,G12117,train,0b971b613f074e21f474e3d0d8c306c8,d878c51613c55e1f
Normal-3259.png,Normal,Normal/images/Normal-3259.png,Normal/masks/Normal-3259.png,True,G12118,test,7326a9a91b36e7d59161c7ca493f8383,9c2dc31b4b451e7a
Normal-326.png,Normal,Normal/images/Normal-326.png,Normal/masks/Normal-326.png,True,G12119,train,82a0f1dd36f8ea2d1d3f652ce5a900f6,91a4c639ec6d38ce
Normal-3260.png,Normal,Normal/images/Normal-3260.png,Normal/masks/Normal-3260.png,True,G12120,train,1645081754fa1bbf189f14d725f7e99a,dd3ac21f4c1b1ae4
Normal-3261.png,Normal,Normal/images/Normal-3261.png,Normal/masks/Normal-3261.png,True,G12121,train,edb1dd62eae769149639ff7dfd37f310,9c3dc01793431f5e
Normal-3262.png,Normal,Normal/images/Normal-3262.png,Normal/masks/Normal-3262.png,True,G12122,train,9ea2e013fa01732513fbd8e08cb9a3f2,887f061a4f7a0d5d
Normal-3263.png,Normal,Normal/images/Normal-3263.png,Normal/masks/Normal-3263.png,True,G12123,train,71e9179c08ce2c5164b164cb8a8f5373,d1390bdc1d58f929
Normal-3264.png,Normal,Normal/images/Normal-3264.png,Normal/masks/Normal-3264.png,True,G12124,train,f521f5573df3baa81ef76f8ca3110270,d13d4206df1b9c4e
Normal-3265.png,Normal,Normal/images/Normal-3265.png,Normal/masks/Normal-3265.png,True,G12125,train,155bfc7b0150ab19ae6bd6ea7482d1f0,dd29861717071f1d
Normal-3266.png,Normal,Normal/images/Normal-3266.png,Normal/masks/Normal-3266.png,True,G12126,train,1a2d949094a2520df0244ca7721eb79b,c029d6d65a5a4e4f
Normal-3267.png,Normal,Normal/images/Normal-3267.png,Normal/masks/Normal-3267.png,True,G12127,train,0318d651c9512115343e413e04be77cc,9979071a917879e9
Normal-3268.png,Normal,Normal/images/Normal-3268.png,Normal/masks/Normal-3268.png,True,G12128,train,af087ca2f3663cb029e42c06b30d7cb4,d03d024f393ee5e1
Normal-3269.png,Normal,Normal/images/Normal-3269.png,Normal/masks/Normal-3269.png,True,G12129,train,35ba23b864d8c94484bbed6c3cf0b199,9968061a555f7975
Normal-327.png,Normal,Normal/images/Normal-327.png,Normal/masks/Normal-327.png,True,G12130,train,0275abe971dcb00429da164397ff4b1e,9139c33b997938f0
Normal-3270.png,Normal,Normal/images/Normal-3270.png,Normal/masks/Normal-3270.png,True,G12131,val,a43f70dc5b12f250df0e070d3d7c3ed5,887f065e0c4f5b4d
Normal-3271.png,Normal,Normal/images/Normal-3271.png,Normal/masks/Normal-3271.png,True,G12132,val,4e655640f78a64e5a170b1ea51de411e,9805065b5d7eeab8
Normal-3272.png,Normal,Normal/images/Normal-3272.png,Normal/masks/Normal-3272.png,True,G12133,test,a68edb69965c7f0adf9d72ff11a29392,d969061f16471a5f
Normal-3273.png,Normal,Normal/images/Normal-3273.png,Normal/masks/Normal-3273.png,True,G12134,train,3c4cc47de1e251c4afe8a7229ea04135,947e035b0d4d5c5d
Normal-3274.png,Normal,Normal/images/Normal-3274.png,Normal/masks/Normal-3274.png,True,G12135,test,e9f6a6d14f864492c5045b950963015e,c879c65a5e455959
Normal-3275.png,Normal,Normal/images/Normal-3275.png,Normal/masks/Normal-3275.png,True,G12136,train,8b94f46de52694ed6a6ddc5f44dd5b75,d43c43c34b2ae2ed
Normal-3276.png,Normal,Normal/images/Normal-3276.png,Normal/masks/Normal-3276.png,True,G12137,train,2d726931196414a7eab858dfab4c02ee,887dc7c6187d4f18
Normal-3277.png,Normal,Normal/images/Normal-3277.png,Normal/masks/Normal-3277.png,True,G12138,train,019281abb833b7494dbaf145636c8ca1,996d065a497879e5
Normal-3278.png,Normal,Normal/images/Normal-3278.png,Normal/masks/Normal-3278.png,True,G12139,train,3f3f4b0f10a6eb16d4a08b626e717877,d979061b960f1a2f
Normal-3279.png,Normal,Normal/images/Normal-3279.png,Normal/masks/Normal-3279.png,True,G12140,val,c2e2b248b0e9471c9771a21d806de09f,d92b161a1c4e5b7a
Normal-328.png,Normal,Normal/images/Normal-328.png,Normal/masks/Normal-328.png,True,G12141,train,780de8b64567d59140711788575681a5,9d3f821b1d5c1cd4
Normal-3280.png,Normal,Normal/images/Normal-3280.png,Normal/masks/Normal-3280.png,True,G12142,train,eaa9a3a1608217b39e7197904a4b6860,997fc61a1b451f30
Normal-3281.png,Normal,Normal/images/Normal-3281.png,Normal/masks/Normal-3281.png,True,G12143,val,934460ffa03ed00d18d7023591147b86,9d2cd21ad7431b53
Normal-3282.png,Normal,Normal/images/Normal-3282.png,Normal/masks/Normal-3282.png,True,G12144,test,01a42dc7c29b6eee69813769b4c413a3,9968075a1b6b7be0
Normal-3283.png,Normal,Normal/images/Normal-3283.png,Normal/masks/Normal-3283.png,True,G12145,test,815fdf259ab5caa4485b564904a236bb,996d061a4d7af04f
Normal-3284.png,Normal,Normal/images/Normal-3284.png,Normal/masks/Normal-3284.png,True,G12146,train,11bc4fe33444b060671dc99a65539d4f,d979d4421b570f19
Normal-3285.png,Normal,Normal/images/Normal-3285.png,Normal/masks/Normal-3285.png,True,G12147,train,4c3b3e1b42d9323bf5207eae41b32b0e,9979c686149acfb8
Normal-3286.png,Normal,Normal/images/Normal-3286.png,Normal/masks/Normal-3286.png,True,G12148,train,5fbb5b7b0ac7a081a99f923b93746af7,d46c430f38ee3cf0
Normal-3287.png,Normal,Normal/images/Normal-3287.png,Normal/masks/Normal-3287.png,True,G12149,train,02601785e46ce5958f76e623dba93379,d02fd2575e504a5d
Normal-3288.png,Normal,Normal/images/Normal-3288.png,Normal/masks/Normal-3288.png,True,G12150,train,88d98b5b45b11e6834d4a7dbd865401c,d878d29a4dca59c9
Normal-3289.png,Normal,Normal/images/Normal-3289.png,Normal/masks/Normal-3289.png,True,G12151,train,6ec4523217b26f12ca8e2309f9619c24,8979061e164f5f3a
Normal-329.png,Normal,Normal/images/Normal-329.png,Normal/masks/Normal-329.png,True,G12152,train,9499f363bb8fdb6971733f0de0dacd49,957c873b8d3c681c
Normal-3290.png,Normal,Normal/images/Normal-3290.png,Normal/masks/Normal-3290.png,True,G12153,train,95551dbc6735cfcbe3ca37f18bee7925,dd3cc24b1b406b9e
Normal-3291.png,Normal,Normal/images/Normal-3291.png,Normal/masks/Normal-3291.png,True,G12154,train,bb2ddadc5b921ef141edb853d0b1ed4d,d969c71ec51ed403
Normal-3292.png,Normal,Normal/images/Normal-3292.png,Normal/masks/Normal-3292.png,True,G12155,val,0893405aa9cd26c12708976619772570,d12953431f1cbd1b
Normal-3293.png,Normal,Normal/images/Normal-3293.png,Normal/masks/Normal-3293.png,True,G12156,test,c6cb17cfad75979a6cf9d0f7403b9b0d,c43dc34d0a692f6e
Normal-3294.png,Normal,Normal/images/Normal-3294.png,Normal/masks/Normal-3294.png,True,G12157,train,7cf91ffbf39daf7d210bcff7ac1a1f79,d84987161c5f3af8
Normal-3295.png,Normal,Normal/images/Normal-3295.png,Normal/masks/Normal-3295.png,True,G12158,train,ff99361e6d372531370229ce30adef43,dc3dc319114f3d51
Normal-3296.png,Normal,Normal/images/Normal-3296.png,Normal/masks/Normal-3296.png,True,G12159,train,fba1e07cfe57065da6954d534fc9441a,897985165a1f575a
Normal-3297.png,Normal,Normal/images/Normal-3297.png,Normal/masks/Normal-3297.png,True,G12160,train,09b8aa261c51243452c3b6c76de6ba6e,dc3fc25b50413bcc
Normal-3298.png,Normal,Normal/images/Normal-3298.png,Normal/masks/Normal-3298.png,True,G12161,train,9d5a531304507d95396ad29da479a6e6,996f061a471e1d4f
Normal-3299.png,Normal,Normal/images/Normal-3299.png,Normal/masks/Normal-3299.png,True,G12162,train,07ceb848346496b5458b850843014295,9de96e1661e66462
Normal-33.png,Normal,Normal/images/Normal-33.png,Normal/masks/Normal-33.png,True,G12163,train,9c36a52469c75d5d85c77adb76e064e1,d03d861b1c4d3d2f
Normal-330.png,Normal,Normal/images/Normal-330.png,Normal/masks/Normal-330.png,True,G12164,train,c6076577da832d1b6e6cbb565bd4110e,9129c26bd9697879
Normal-3300.png,Normal,Normal/images/Normal-3300.png,Normal/masks/Normal-3300.png,True,G12165,train,20047338a75680c5a4c1b1f9e6924db7,dd3cc25f944f1948
Normal-3301.png,Normal,Normal/images/Normal-3301.png,Normal/masks/Normal-3301.png,True,G12166,train,1c06141b3a8d4f05437edad2b017fe31,993f06185f184d5f
Normal-3302.png,Normal,Normal/images/Normal-3302.png,Normal/masks/Normal-3302.png,True,G12167,train,6cbd283dc676059292ad68c628a36be3,91681e1b1f3ab9a5
Normal-3303.png,Normal,Normal/images/Normal-3303.png,Normal/masks/Normal-3303.png,True,G12168,train,631001debd99b14f6ba0107b45f0ca09,d928c61b5c4a3b6d
Normal-3304.png,Normal,Normal/images/Normal-3304.png,Normal/masks/Normal-3304.png,True,G12169,val,687f9ab5fa063d4a4a80d7ebe1c4578c,d93b46191929b9b9
Normal-3305.png,Normal,Normal/images/Normal-3305.png,Normal/masks/Normal-3305.png,True,G12170,val,d8da8a4f07e07df52447ce3faca02ebc,9d2ec21b4d4f1b15
Normal-3306.png,Normal,Normal/images/Normal-3306.png,Normal/masks/Normal-3306.png,True,G12171,train,0d721d6895ca313dc664833b1c78c5b4,8169861a195e7b7d
Normal-3307.png,Normal,Normal/images/Normal-3307.png,Normal/masks/Normal-3307.png,True,G12172,val,3d8fc7f75c3636a58c559ecc842985d9,91788a4f19196d7b
Normal-3308.png,Normal,Normal/images/Normal-3308.png,Normal/masks/Normal-3308.png,True,G12173,test,55e39f53334dcf304ee87273e7e3c642,d93946071e1f4f1a
Normal-3309.png,Normal,Normal/images/Normal-3309.png,Normal/masks/Normal-3309.png,True,G12174,test,a8e0875be53d3e434757b615696f2521,d959c65a194b1b1b
Normal-331.png,Normal,Normal/images/Normal-331.png,Normal/masks/Normal-331.png,True,G12175,train,2ba04d688e860aa21243687d88bebbf2,957c871b8d4d4cc5
Normal-3310.png,Normal,Normal/images/Normal-3310.png,Normal/masks/Normal-3310.png,True,G12176,train,f6157e9fe9823cf690c97a5a13880907,8979c71e144e5b59
Normal-3311.png,Normal,Normal/images/Normal-3311.png,Normal/masks/Normal-3311.png,True,G12177,train,dc6f50cb5e25a996d283cec8e82ada58,d939d2171e4a9f48
Normal-3312.png,Normal,Normal/images/Normal-3312.png,Normal/masks/Normal-3312.png,True,G12178,test,6d3433b9385bcfd9f47cc5f0432126f6,d03d42433d0fce1f
Normal-3313.png,Normal,Normal/images/Normal-3313.png,Normal/masks/Normal-3313.png,True,G12179,train,6d078c94666ff4d1a3d0360a48780fd1,d13d870e1a516f3a
Normal-3314.png,Normal,Normal/images/Normal-3314.png,Normal/masks/Normal-3314.png,True,G12180,train,9f390c160bd00a65a5b3664b9a2080a0,887807971ac71f9b
Normal-3315.png,Normal,Normal/images/Normal-3315.png,Normal/masks/Normal-3315.png,True,G12181,train,f2d148e3e505918ee17321207c8b686a,b92c461b1f0e1b1f
Normal-3316.png,Normal,Normal/images/Normal-3316.png,Normal/masks/Normal-3316.png,True,G12182,train,92c0d34c049743591d911a925c579587,dd2c821f545f1d4c
Normal-3317.png,Normal,Normal/images/Normal-3317.png,Normal/masks/Normal-3317.png,True,G12183,train,2c49d7b67af496d4d166a7b7cfdc0bbb,c92c465b174e594f
Normal-3318.png,Normal,Normal/images/Normal-3318.png,Normal/masks/Normal-3318.png,True,G12184,train,f8f5da42c373975f5a4c1b2b40bf021e,d93dc65b144a1b5c
Normal-3319.png,Normal,Normal/images/Normal-3319.png,Normal/masks/Normal-3319.png,True,G12185,train,96f424b4a6d76caedfa62ea8a3115a68,d93ac21b1c0e3dea
Normal-332.png,Normal,Normal/images/Normal-332.png,Normal/masks/Normal-332.png,True,G12186,train,21ac3eef7dbcf923d0ef2d7e1b0fd839,816ac67aec69394d
Normal-3320.png,Normal,Normal/images/Normal-3320.png,Normal/masks/Normal-3320.png,True,G12187,train,32cc867c7e0105581ecd9b5f592cd3a6,dc6ac31f4c0d1c4f
Normal-3321.png,Normal,Normal/images/Normal-3321.png,Normal/masks/Normal-3321.png,True,G12188,train,6053ef364de4ed6f5f381dce0625ccc8,dc3d42030b8f2fbc
Normal-3322.png,Normal,Normal/images/Normal-3322.png,Normal/masks/Normal-3322.png,True,G12189,train,d7eb69f944944a0891f48043b8ce37c9,dc2e42570d4abf1c
Normal-3323.png,Normal,Normal/images/Normal-3323.png,Normal/masks/Normal-3323.png,True,G12190,val,82658399a802943912cc4278374adccd,d13d4a561919b4af
Normal-3324.png,Normal,Normal/images/Normal-3324.png,Normal/masks/Normal-3324.png,True,G12191,val,3d0bf4c03720e717fbf2867173d27399,d52e43434b4d4f4e
Normal-3325.png,Normal,Normal/images/Normal-3325.png,Normal/masks/Normal-3325.png,True,G12192,val,76fca72433cdae3d0e41a14430c2ca97,817a561a59497bcd
Normal-3326.png,Normal,Normal/images/Normal-3326.png,Normal/masks/Normal-3326.png,True,G12193,test,718404f19a9a82039c5826abb61aedf5,d978d61a5d405a5d
Normal-3327.png,Normal,Normal/images/Normal-3327.png,Normal/masks/Normal-3327.png,True,G12194,test,4e7b8a521f57317d3a1aa92bcb840e35,897886185d7b39c7
Normal-3328.png,Normal,Normal/images/Normal-3328.png,Normal/masks/Normal-3328.png,True,G12195,train,aebaa69f6a2f498cb19b96f6b9769ba8,dd28d65397425b58
Normal-3329.png,Normal,Normal/images/Normal-3329.png,Normal/masks/Normal-3329.png,True,G12196,test,b37c93db25c1c7705d004ef15bc01b41,91689e0a1b5d5d79
Normal-333.png,Normal,Normal/images/Normal-333.png,Normal/masks/Normal-333.png,True,G12197,train,ee93b2c28cde5f16845a91aa36ea8f4a,d13dc71aad293c4c
Normal-3330.png,Normal,Normal/images/Normal-3330.png,Normal/masks/Normal-3330.png,True,G12198,train,850a53dba3ffa7063b786f20c2c40737,d83946161e1deb1d
Normal-3331.png,Normal,Normal/images/Normal-3331.png,Normal/masks/Normal-3331.png,True,G12199,train,5e62562aa67f5b1b1a64f6b42d45b804,dc2fc0461e4d691f
Normal-3332.png,Normal,Normal/images/Normal-3332.png,Normal/masks/Normal-3332.png,True,G12200,train,91e21586b05004f7a865dc256ae66bcf,d95a161a180f3ebe
Normal-3333.png,Normal,Normal/images/Normal-3333.png,Normal/masks/Normal-3333.png,True,G12201,train,1af45358b6dd81e884688d980c2e199e,c92dd61bd44f5944
Normal-3334.png,Normal,Normal/images/Normal-3334.png,Normal/masks/Normal-3334.png,True,G12202,train,ab679ff1475965690338308703cb3194,dd59525b1f081a3e
Normal-3335.png,Normal,Normal/images/Normal-3335.png,Normal/masks/Normal-3335.png,True,G12203,train,4774ab1d0f86b2b03304df356f49fa9a,dc2e430f486d3e78
Normal-3336.png,Normal,Normal/images/Normal-3336.png,Normal/masks/Normal-3336.png,True,G12204,train,cc07d81aeb6c90762a5547e878f916f6,dd2c02c3593cecea
Normal-3337.png,Normal,Normal/images/Normal-3337.png,Normal/masks/Normal-3337.png,True,G12205,train,56e0e8a6427a537d7d9d8b9a913921aa,982dc4061f495dfe
Normal-3338.png,Normal,Normal/images/Normal-3338.png,Normal/masks/Normal-3338.png,True,G12206,train,3847cb761a73f371974317c4036d5bc2,997d46521b45471f
Normal-3339.png,Normal,Normal/images/Normal-3339.png,Normal/masks/Normal-3339.png,True,G12207,train,e82fc9b398dcc6c1bd078a8895cf1376,987d875e1a0d5f18
Normal-334.png,Normal,Normal/images/Normal-334.png,Normal/masks/Normal-334.png,True,G12208,train,0a508d07b98e2dea639984521627873e,986fc71eca4b2c4c
Normal-3340.png,Normal,Normal/images/Normal-3340.png,Normal/masks/Normal-3340.png,True,G12209,val,66d0a25c5e3e52d8c92478fdff2f8c5b,813dd21f5e5a585a
Normal-3341.png,Normal,Normal/images/Normal-3341.png,Normal/masks/Normal-3341.png,True,G12210,train,535f23e4502097c51fb8ac512d610fc6,dd2e504b0f442e5f
Normal-3342.png,Normal,Normal/images/Normal-3342.png,Normal/masks/Normal-3342.png,True,G12211,train,0a8a386b059849330338aedc7e00fe27,895f061a5f185e7a
Normal-3343.png,Normal,Normal/images/Normal-3343.png,Normal/masks/Normal-3343.png,True,G12212,train,e2e23991ff0ee2e7ea3b1180de8a04a4,8178c616fa1ff819
Normal-3344.png,Normal,Normal/images/Normal-3344.png,Normal/masks/Normal-3344.png,True,G12213,train,3fe61ec1cc300f73f6fdc6c9c4e61233,e93907461a1ede5a
Normal-3345.png,Normal,Normal/images/Normal-3345.png,Normal/masks/Normal-3345.png,True,G12214,train,907251a0ffd436b6f55cb05d76b111b8,892d465a0c6bbced
Normal-3346.png,Normal,Normal/images/Normal-3346.png,Normal/masks/Normal-3346.png,True,G12215,train,161066e89bfa4ee24507386aaeb3e7ec,816947165a975e9b
Normal-3347.png,Normal,Normal/images/Normal-3347.png,Normal/masks/Normal-3347.png,True,G12216,train,be48a29855fd13236e8a79a10e4f8b33,d9bd861b05461fc6
Normal-3348.png,Normal,Normal/images/Normal-3348.png,Normal/masks/Normal-3348.png,True,G12217,train,acf378a3876675f6bd0870b8ef5df0a2,d87b85961e195b1a
Normal-3349.png,Normal,Normal/images/Normal-3349.png,Normal/masks/Normal-3349.png,True,G12218,val,cd49ab84db64b45109720615e7a86cf4,917a861a1b5f1b1b
Normal-335.png,Normal,Normal/images/Normal-335.png,Normal/masks/Normal-335.png,True,G12219,val,615a8f7bfc9003a370f5d057796cde7b,902be27ade4a5d68
Normal-3350.png,Normal,Normal/images/Normal-3350.png,Normal/masks/Normal-3350.png,True,G12220,test,be75fdb428e8647c34bf07a1e0d63c94,89eb061e471a1dcf
Normal-3351.png,Normal,Normal/images/Normal-3351.png,Normal/masks/Normal-3351.png,True,G12221,test,2de17135783ac4b94db77a005e3275cd,97685467a39c5a63
Normal-3352.png,Normal,Normal/images/Normal-3352.png,Normal/masks/Normal-3352.png,True,G12222,test,6b67b7936a1469c5e17d3ed96c16b414,997f061a59e879a1
Normal-3353.png,Normal,Normal/images/Normal-3353.png,Normal/masks/Normal-3353.png,True,G12223,test,68b9f6dc6de61e164584a880e9adc004,d17b56565e4c4b18
Normal-3354.png,Normal,Normal/images/Normal-3354.png,Normal/masks/Normal-3354.png,True,G12224,test,521d8d711010b1f53eadd1e937668c61,897b861e1a0d5f5a
Normal-3355.png,Normal,Normal/images/Normal-3355.png,Normal/masks/Normal-3355.png,True,G12225,train,1800f64c0df04d21d91ecb39b7b5cf38,d47e534b5f464a18
Normal-3356.png,Normal,Normal/images/Normal-3356.png,Normal/masks/Normal-3356.png,True,G12226,train,2fa0f2c3add918f4e2a419d36b8e2e36,896d87961f86171e
Normal-3357.png,Normal,Normal/images/Normal-3357.png,Normal/masks/Normal-3357.png,True,G12227,train,18c9d90f8141845e179ec3c327bb8a7d,817b461a1ddb7a89
Normal-3358.png,Normal,Normal/images/Normal-3358.png,Normal/masks/Normal-3358.png,True,G12228,train,4ab4706c78d1626d8bbb2eab5d53ed46,987b07861e951e5e
Normal-3359.png,Normal,Normal/images/Normal-3359.png,Normal/masks/Normal-3359.png,True,G12229,train,047b8a2c0337067a5d800598b7ca8b36,dc3c830b1bc26f3c
Normal-336.png,Normal,Normal/images/Normal-336.png,Normal/masks/Normal-336.png,True,G12230,train,3f2438a9b1498f8b4deb9249e2d14e74,952f0f1c4d5d48ce
Normal-3360.png,Normal,Normal/images/Normal-3360.png,Normal/masks/Normal-3360.png,True,G12231,train,09a0c2a9bd8410d981f17773b17e6ffe,807b47861e18fb5b
Normal-3361.png,Normal,Normal/images/Normal-3361.png,Normal/masks/Normal-3361.png,True,G12232,val,4bca837deb1833490937f94b42d322a2,dc2d525f424b1f43
Normal-3362.png,Normal,Normal/images/Normal-3362.png,Normal/masks/Normal-3362.png,True,G12233,val,a099e103b807b269ed80aa0bdf8b5d4d,c83d05d65a586dce
Normal-3363.png,Normal,Normal/images/Normal-3363.png,Normal/masks/Normal-3363.png,True,G12234,train,1d527bd90445a987d4c04f184390d78e,d52f021d1a6d6b39
Normal-3364.png,Normal,Normal/images/Normal-3364.png,Normal/masks/Normal-3364.png,True,G12235,train,87bdf739c6b2f531bcd8b3a06b331e23,dd2c831b454e594f
Normal-3365.png,Normal,Normal/images/Normal-3365.png,Normal/masks/Normal-3365.png,True,G12236,val,a8d0dd3efe6ebd959159f9830a8e221e,9979841e1ecc1b3d
Normal-3366.png,Normal,Normal/images/Normal-3366.png,Normal/masks/Normal-3366.png,True,G12237,test,6b5a74d94ec3474a03c85e5db68ac653,dcafc35944691d4c
Normal-3367.png,Normal,Normal/images/Normal-3367.png,Normal/masks/Normal-3367.png,True,G12238,train,4e660293dfcfe73032a1f89b26750904,dc2e021d4eee39e4
Normal-3368.png,Normal,Normal/images/Normal-3368.png,Normal/masks/Normal-3368.png,True,G12239,train,bfc5ef3e0c94b858c45e20f9f7eb2a9e,d93d52171c0a3f39
Normal-3369.png,Normal,Normal/images/Normal-3369.png,Normal/masks/Normal-3369.png,True,G12240,train,75c4bd380fbac7ff366d0e207cf69da4,dd3d42035f0a6c3d
Normal-337.png,Normal,Normal/images/Normal-337.png,Normal/masks/Normal-337.png,True,G12241,train,da90f320b7e8952f9c4426abd9afff21,917cc71a9d4d19cc
Normal-3370.png,Normal,Normal/images/Normal-3370.png,Normal/masks/Normal-3370.png,True,G12242,train,ff31652a336d9822e42c49776c70e2c7,d57f431e4a0f180f
Normal-3371.png,Normal,Normal/images/Normal-3371.png,Normal/masks/Normal-3371.png,True,G12243,train,93077b02737fbd6a21f50b9804a0ceca,fd3cc20b0c6a3de4
Normal-3372.png,Normal,Normal/images/Normal-3372.png,Normal/masks/Normal-3372.png,True,G12244,train,bbed8cc053c350c0a9d8dde82c64ef02,d91d861a1d4b396b
Normal-3373.png,Normal,Normal/images/Normal-3373.png,Normal/masks/Normal-3373.png,True,G12245,train,8e066e10b6a5734d62c2ec07e5a7db7b,817f861e185f5b19
Normal-3374.png,Normal,Normal/images/Normal-3374.png,Normal/masks/Normal-3374.png,True,G12246,train,8278d53dd0a5d20ce53dd47ea0fcadbf,85682f073e5b68b6
Normal-3375.png,Normal,Normal/images/Normal-3375.png,Normal/masks/Normal-3375.png,True,G12247,train,a1979ea483d7ca671791a0b3d609999a,d93c461a194f3cb9
Normal-3376.png,Normal,Normal/images/Normal-3376.png,Normal/masks/Normal-3376.png,True,G12248,train,fce9506f3080025bad7509283adce2c4,997fc61b190b1939
Normal-3377.png,Normal,Normal/images/Normal-3377.png,Normal/masks/Normal-3377.png,True,G12249,train,e1912e13028cb03d744f1d76f0482847,d1790606185f6f3b
Normal-3378.png,Normal,Normal/images/Normal-3378.png,Normal/masks/Normal-3378.png,True,G12250,val,0a4f629d6d3b950de0f4a2c2f81f3a7b,997cc7521b454f1a
Normal-3379.png,Normal,Normal/images/Normal-3379.png,Normal/masks/Normal-3379.png,True,G12251,train,c79700a96d7be8d5ab85b2e23394fa7f,d169ce11194e3b1f
Normal-338.png,Normal,Normal/images/Normal-338.png,Normal/masks/Normal-338.png,True,G12252,train,37bcae180ee3d24dffcac861c8880fb1,d47ec72f8d2c2c44
Normal-3380.png,Normal,Normal/images/Normal-3380.png,Normal/masks/Normal-3380.png,True,G12253,train,72c5b12fc1512fa886d5fe97cd01faa9,dc3c074b1a4c6fb8
Normal-3381.png,Normal,Normal/images/Normal-3381.png,Normal/masks/Normal-3381.png,True,G12254,train,a2d901f9035fc1a366fe6ef390a42025,997bc61e584b1a4b
Normal-3382.png,Normal,Normal/images/Normal-3382.png,Normal/masks/Normal-3382.png,True,G12255,test,c786fa832cafba8f5f252ffe9571b640,d03d464b196d7cb8
Normal-3383.png,Normal,Normal/images/Normal-3383.png,Normal/masks/Normal-3383.png,True,G12256,test,0b058bf17f485749a308bc22a021a89b,8879c71b94465bd6
Normal-3384.png,Normal,Normal/images/Normal-3384.png,Normal/masks/Normal-3384.png,True,G12257,val,c4e5069d59e5ea4ebe76e9ec0a38c381,91295e16585bed39
Normal-3385.png,Normal,Normal/images/Normal-3385.png,Normal/masks/Normal-3385.png,True,G12258,val,1ff0497b0bee2f71d82577193148e876,d43d43d31f4fc038
Normal-3386.png,Normal,Normal/images/Normal-3386.png,Normal/masks/Normal-3386.png,True,G12259,test,363c4f688a948c60195b6349d03aab3a,dd2e021c4f3ac1ee
Normal-3387.png,Normal,Normal/images/Normal-3387.png,Normal/masks/Normal-3387.png,True,G12260,train,12e55ebc02babb6811e376aaeed86d1f,8c6dc35b5d451e54
Normal-3388.png,Normal,Normal/images/Normal-3388.png,Normal/masks/Normal-3388.png,True,G12261,train,de775f0bd2c472257ec2ff9a1e456f24,c13ad6083b3b7b31
Normal-3389.png,Normal,Normal/images/Normal-3389.png,Normal/masks/Normal-3389.png,True,G12262,train,ec15fb6fcbe212b2389627b7d40ce08d,917ac7521a5f461b
Normal-339.png,Normal,Normal/images/Normal-339.png,Normal/masks/Normal-339.png,True,G12263,train,5efc2aeabb30ac84c64840d23063e11e,917cc61a989d3e96
Normal-3390.png,Normal,Normal/images/Normal-3390.png,Normal/masks/Normal-3390.png,True,G12264,train,828c235bc0f8d6b3b615365898680bc1,dc3dc21b81cf3d90
Normal-3391.png,Normal,Normal/images/Normal-3391.png,Normal/masks/Normal-3391.png,True,G12265,train,fac994d5525b356c1585131b3b592308,99790e1a195e79b1
Normal-3392.png,Normal,Normal/images/Normal-3392.png,Normal/masks/Normal-3392.png,True,G12266,train,bfdc89f046c389e7f712aeca41b99b38,a86bc61a1bc65b39
Normal-3393.png,Normal,Normal/images/Normal-3393.png,Normal/masks/Normal-3393.png,True,G12267,train,e7d5615a50c253a9e68d268738c63539,817e075b0d4f1b3c
Normal-3394.png,Normal,Normal/images/Normal-3394.png,Normal/masks/Normal-3394.png,True,G12268,train,0628fefe8e602affc5c64da6e7d15d2a,985d061a18ff79e8
Normal-3395.png,Normal,Normal/images/Normal-3395.png,Normal/masks/Normal-3395.png,True,G12269,train,1ee0435e3f4cbc7d278d76984435b192,9978061a196e7b79
Normal-3396.png,Normal,Normal/images/Normal-3396.png,Normal/masks/Normal-3396.png,True,G12270,train,5a94555163c1495119f8a058a5fa521f,9978065278e8e5f9
Normal-3397.png,Normal,Normal/images/Normal-3397.png,Normal/masks/Normal-3397.png,True,G12271,train,0e02d9cc2c3ba7c4a3e58793e8027682,996886421f4e3f5b
Normal-3398.png,Normal,Normal/images/Normal-3398.png,Normal/masks/Normal-3398.png,True,G12272,val,1b293557f8c65f4719a2b3b966a68560,b97dc4d2d2c51a51
Normal-3399.png,Normal,Normal/images/Normal-3399.png,Normal/masks/Normal-3399.png,True,G12273,val,0493729928e603d6e673d380bcb1963d,c868d2da5b4d5c1e
Normal-34.png,Normal,Normal/images/Normal-34.png,Normal/masks/Normal-34.png,True,G12274,val,af99ccbccdfd1c798337014f5a2aff9f,c93d461b0a6f38ae
Normal-340.png,Normal,Normal/images/Normal-340.png,Normal/masks/Normal-340.png,True,G12275,test,f5fe1a666481c481aeb97a54fc2f8dd7,907e565a1d193997
Normal-3400.png,Normal,Normal/images/Normal-3400.png,Normal/masks/Normal-3400.png,True,G12276,test,a67cd3ebb969f0a0495d46e1332927ee,992806179f4f1b9e
Normal-3401.png,Normal,Normal/images/Normal-3401.png,Normal/masks/Normal-3401.png,True,G12277,test,02c9e42431deef431eac623b4e12033b,d97817071a5e0f5a
Normal-3402.png,Normal,Normal/images/Normal-3402.png,Normal/masks/Normal-3402.png,True,G12278,train,bebf46936a9f5d9a3c790e9ed40f3172,dc2d42d20bcb6fd0
Normal-3403.png,Normal,Normal/images/Normal-3403.png,Normal/masks/Normal-3403.png,True,G12279,train,1882feb2c680a1e502c8090fe1de5d68,9168c61ede5a585b
Normal-3404.png,Normal,Normal/images/Normal-3404.png,Normal/masks/Normal-3404.png,True,G12280,val,03d1246ddebaa7fc68e7fe3573cd9fa6,cd2dd2021f4a1f2f
Normal-3405.png,Normal,Normal/images/Normal-3405.png,Normal/masks/Normal-3405.png,True,G12281,train,ac179984507bb4022957eb7766e40f24,803f570a5f393b29
Normal-3406.png,Normal,Normal/images/Normal-3406.png,Normal/masks/Normal-3406.png,True,G12282,train,d0de5c26b07c0b48d32ad5a2ab6df747,d83d530b0e2a2f3d
Normal-3407.png,Normal,Normal/images/Normal-3407.png,Normal/masks/Normal-3407.png,True,G12283,train,1003e4718e481f4fc2469e6593ace04b,943d07871b0e4f1f
Normal-3408.png,Normal,Normal/images/Normal-3408.png,Normal/masks/Normal-3408.png,True,G12284,train,fa475de193d2ab61b218f13639eea47c,cd7cd2cb4dc81989
Normal-3409.png,Normal,Normal/images/Normal-3409.png,Normal/masks/Normal-3409.png,True,G12285,train,b41f4b7d46b214c67f564cb8ade52dc8,d12cde06dc0bfc19
Normal-341.png,Normal,Normal/images/Normal-341.png,Normal/masks/Normal-341.png,True,G12286,test,e8589a535b3c9914b76585ec6b75cdb0,917ea658c9854dc7
Normal-3410.png,Normal,Normal/images/Normal-3410.png,Normal/masks/Normal-3410.png,True,G12287,train,011079322e3dc03d445540fa83a45629,dd3d421d920f1ce9
Normal-3411.png,Normal,Normal/images/Normal-3411.png,Normal/masks/Normal-3411.png,True,G12288,train,7c9bcc11cfdb915f51e557f44103916d,9968071e58cd78f9
Normal-3412.png,Normal,Normal/images/Normal-3412.png,Normal/masks/Normal-3412.png,True,G12289,test,a55f8bcddf79c04a64638b6decfa4fb6,81e8c618871cf7de
Normal-3413.png,Normal,Normal/images/Normal-3413.png,Normal/masks/Normal-3413.png,True,G12290,train,8275fdf2c9f4b3a4f458becb0b75e46c,8c2cd34b5d4e1d5c
Normal-3414.png,Normal,Normal/images/Normal-3414.png,Normal/masks/Normal-3414.png,True,G12291,train,5e51fec9783541d696ce753b08471f5f,dd3d525f18430f1a
Normal-3415.png,Normal,Normal/images/Normal-3415.png,Normal/masks/Normal-3415.png,True,G12292,train,3cf61409060625b9c2983d59a4995c17,83691c1c1b5b5b79
Normal-3416.png,Normal,Normal/images/Normal-3416.png,Normal/masks/Normal-3416.png,True,G12293,train,e5eba05231602c041e2fd9dc6f30ace1,dc3dc2435bc20f1b
Normal-3417.png,Normal,Normal/images/Normal-3417.png,Normal/masks/Normal-3417.png,True,G12294,train,317b9306a904a862a425be815ba7ab49,dc3d4253190d0fbd
Normal-3418.png,Normal,Normal/images/Normal-3418.png,Normal/masks/Normal-3418.png,True,G12295,val,70d39f4aa9a40923be833089301d89da,d93bc41f160e1e4e
Normal-3419.png,Normal,Normal/images/Normal-3419.png,Normal/masks/Normal-3419.png,True,G12296,train,a0dc658dc468ffb0138e75b7c5ec6fbd,dd2cc2571d441e3d
Normal-342.png,Normal,Normal/images/Normal-342.png,Normal/masks/Normal-342.png,True,G12297,test,958e5b5b8eba72a55c98dfd45a9532c3,917d8e3a8d0d354d
Normal-3420.png,Normal,Normal/images/Normal-3420.png,Normal/masks/Normal-3420.png,True,G12298,val,b1af833b2f80cb77ba29c73fd1aeaa73,d969861a154e1b5f
Normal-3421.png,Normal,Normal/images/Normal-3421.png,Normal/masks/Normal-3421.png,True,G12299,train,66272669ba52ee07646f304fd58fe2af,dd3cc25b0a4e2b3c
Normal-3422.png,Normal,Normal/images/Normal-3422.png,Normal/masks/Normal-3422.png,True,G12300,train,04aaaf2ddc8cd3afba0e8024916e7c02,816956165a5f697a
Normal-3423.png,Normal,Normal/images/Normal-3423.png,Normal/masks/Normal-3423.png,True,G12301,train,9d5aed83d066e5a200efc1b5a657487a,cc7d4a0c959ec3d4
Normal-3424.png,Normal,Normal/images/Normal-3424.png,Normal/masks/Normal-3424.png,True,G12302,val,4c9e5ce96873220846becd019405db67,887b471a1d4daa97
Normal-3425.png,Normal,Normal/images/Normal-3425.png,Normal/masks/Normal-3425.png,True,G12303,train,f48183a57da4cba944d5c5d31bb1f6ca,dc2d821ac71e1d4f
Normal-3426.png,Normal,Normal/images/Normal-3426.png,Normal/masks/Normal-3426.png,True,G12304,train,b44ede90b0d2dce00d05e6b400de4185,993b8616134f1abd
Normal-3427.png,Normal,Normal/images/Normal-3427.png,Normal/masks/Normal-3427.png,True,G12305,train,acc84cf0dece82d8e3307ab5668a7a7e,dc2ec30b4d4f0e4e
Normal-3428.png,Normal,Normal/images/Normal-3428.png,Normal/masks/Normal-3428.png,True,G12306,train,56e4ed10071f0d5254a359105cba5648,9968061b471bfd2e
Normal-3429.png,Normal,Normal/images/Normal-3429.png,Normal/masks/Normal-3429.png,True,G12307,train,1d2232ebaccea644813022a7ec0b15fe,883d471a5d4f3d4a
Normal-343.png,Normal,Normal/images/Normal-343.png,Normal/masks/Normal-343.png,True,G12308,train,c612a21d76409a0444a08a7de7f783ed,916ee65ac94d5847
Normal-3430.png,Normal,Normal/images/Normal-3430.png,Normal/masks/Normal-3430.png,True,G12309,train,ed5a07cda6fb302a29171c4fbe2c3ef2,cd2ed2411fca099f
Normal-3431.png,Normal,Normal/images/Normal-3431.png,Normal/masks/Normal-3431.png,True,G12310,train,6a609ce1d8aab522c82b00a4eaed2ca6,9978071a5b0d5e79
Normal-3432.png,Normal,Normal/images/Normal-3432.png,Normal/masks/Normal-3432.png,True,G12311,train,7ac2f8093822f8e4cdac808c98782be4,817f965809fd3ba0
Normal-3433.png,Normal,Normal/images/Normal-3433.png,Normal/masks/Normal-3433.png,True,G12312,train,df2455c5e615744443347c302cf28991,d03cc3471d0f8e3b
Normal-3434.png,Normal,Normal/images/Normal-3434.png,Normal/masks/Normal-3434.png,True,G12313,train,ee0ceb76b70382baacd5b15641c11b16,dd2e861b4c4e38ec
Normal-3435.png,Normal,Normal/images/Normal-3435.png,Normal/masks/Normal-3435.png,True,G12314,val,edb6d0429ae06f826b3e55e11ea3a746,9979871218575f3a
Normal-3436.png,Normal,Normal/images/Normal-3436.png,Normal/masks/Normal-3436.png,True,G12315,val,3c129726761f07232a7f01544ef43974,896c865a4d7c78e5
Normal-3437.png,Normal,Normal/images/Normal-3437.png,Normal/masks/Normal-3437.png,True,G12316,test,3a6c62d086500a3e04cac798883cedd9,992d861a1d5a5ade
Normal-3438.png,Normal,Normal/images/Normal-3438.png,Normal/masks/Normal-3438.png,True,G12317,test,aaa4d225c959f2724ec93afa0e04d7d4,d93c461a196c3aed
Normal-3439.png,Normal,Normal/images/Normal-3439.png,Normal/masks/Normal-3439.png,True,G12318,test,07df8f906a46e50f0cd60aa5b8e35bac,8978571a1f4e3ab1
Normal-344.png,Normal,Normal/images/Normal-344.png,Normal/masks/Normal-344.png,True,G12319,train,43bfa6f313efdf9ca83a659d36f40bfb,947ce71bcf0d6884
Normal-3440.png,Normal,Normal/images/Normal-3440.png,Normal/masks/Normal-3440.png,True,G12320,val,e2145106dea5774eded42433d4e17b74,917d871e198d3e19
Normal-3441.png,Normal,Normal/images/Normal-3441.png,Normal/masks/Normal-3441.png,True,G12321,train,ab8b88cc73476f40d287dcc632f2c81d,dd2cd20b4c4a3dec
Normal-3442.png,Normal,Normal/images/Normal-3442.png,Normal/masks/Normal-3442.png,True,G12322,train,12d6101a1b9258ad9c364c4209ae9420,996ac41e598e1e5b
Normal-3443.png,Normal,Normal/images/Normal-3443.png,Normal/masks/Normal-3443.png,True,G12323,train,2b7adb7dea8b02db3283c4d09cdf4137,dd2a434b1d5d0a5d
Normal-3444.png,Normal,Normal/images/Normal-3444.png,Normal/masks/Normal-3444.png,True,G12324,train,3d1db85cfbff3c9532bf8f6135e9fdcd,9978c61f1b451f38
Normal-3445.png,Normal,Normal/images/Normal-3445.png,Normal/masks/Normal-3445.png,True,G12325,train,695f99811c863cc595b95ab4deed2ff7,c4344b0f9f6e2c9c
Normal-3446.png,Normal,Normal/images/Normal-3446.png,Normal/masks/Normal-3446.png,True,G12326,train,b67bda08151a08be142c047f121cbb29,d929471e1a4f7a19
Normal-3447.png,Normal,Normal/images/Normal-3447.png,Normal/masks/Normal-3447.png,True,G12327,train,26f196964c0b305c97ae74e9c370dbd5,897907161e4f3b3a
Normal-3448.png,Normal,Normal/images/Normal-3448.png,Normal/masks/Normal-3448.png,True,G12328,train,4d8552e50586c14a091f28be362118de,993887161ec51fba
Normal-3449.png,Normal,Normal/images/Normal-3449.png,Normal/masks/Normal-3449.png,True,G12329,train,4b969e339b2638a37ef7b50b92091007,dd2a021f487d38e7
Normal-345.png,Normal,Normal/images/Normal-345.png,Normal/masks/Normal-345.png,True,G12330,train,f1bf13107d5a15afe95dedfee5e5a609,816fe77acc4529c4
Normal-3450.png,Normal,Normal/images/Normal-3450.png,Normal/masks/Normal-3450.png,True,G12331,test,c217f37ccd38f9a0474298841b005f98,dc3cc2431b924f6d
Normal-3451.png,Normal,Normal/images/Normal-3451.png,Normal/masks/Normal-3451.png,True,G12332,train,4627d09246fd4ad8a4ce5bd30c68e901,d97b05161e595f18
Normal-3452.png,Normal,Normal/images/Normal-3452.png,Normal/masks/Normal-3452.png,True,G12333,train,091676d943c163831288ce76a5284813,812857197d9969cf
Normal-3453.png,Normal,Normal/images/Normal-3453.png,Normal/masks/Normal-3453.png,True,G12334,val,bbe136a8a36c6d13630ac48290911cee,cc7cd743595e4249
Normal-3454.png,Normal,Normal/images/Normal-3454.png,Normal/masks/Normal-3454.png,True,G12335,train,06e95d21d98e5b706cd3bf91cd56b5c2,9938861a1f4f3a3d
Normal-3455.png,Normal,Normal/images/Normal-3455.png,Normal/masks/Normal-3455.png,True,G12336,test,aa70e151b2eb075cf121173359076c85,9168579e3a79a538
Normal-3456.png,Normal,Normal/images/Normal-3456.png,Normal/masks/Normal-3456.png,True,G12337,val,61370fa48485062ff717ddcb434cb9cc,8168565918ce2fbf
Normal-3457.png,Normal,Normal/images/Normal-3457.png,Normal/masks/Normal-3457.png,True,G12338,test,dc9e1d81089c01aedd77c938755d5251,887b171e1e09793d
Normal-3458.png,Normal,Normal/images/Normal-3458.png,Normal/masks/Normal-3458.png,True,G12339,train,95aae220a9ab07ac3831542edc37a03b,d939069e1e1c7f18
Normal-3459.png,Normal,Normal/images/Normal-3459.png,Normal/masks/Normal-3459.png,True,G12340,train,d6db620ad5a54c987607a785a94e105a,817e575a1b1d4a1b
Normal-346.png,Normal,Normal/images/Normal-346.png,Normal/masks/Normal-346.png,True,G12341,train,ea3bc73c0e0fa19815fe24304087feb8,d43ec32f9d642ca4
Normal-3460.png,Normal,Normal/images/Normal-3460.png,Normal/masks/Normal-3460.png,True,G12342,train,e9a2d3e54169d89a62e3d1386edc78c7,dca3434b430d4f2f
Normal-3461.png,Normal,Normal/images/Normal-3461.png,Normal/masks/Normal-3461.png,True,G12343,val,5cc9565786526ae3dc03991e26ed407c,8cbcc21bc24fd1c7
Normal-3462.png,Normal,Normal/images/Normal-3462.png,Normal/masks/Normal-3462.png,True,G12344,test,503350f216d5f90aea1c7625f81854d2,d97b05d65a097d18
Normal-3463.png,Normal,Normal/images/Normal-3463.png,Normal/masks/Normal-3463.png,True,G12345,train,0ff16e100d6ff63e3139d8e044e27632,9879c5561e0d5f1a
Normal-3464.png,Normal,Normal/images/Normal-3464.png,Normal/masks/Normal-3464.png,True,G12346,train,67fa51c689892189c7e5f0fb6ee91039,81689618595f7b79
Normal-3465.png,Normal,Normal/images/Normal-3465.png,Normal/masks/Normal-3465.png,True,G12347,train,4d6cb1b86e45bed552168c0c1ed1d0e4,887d461b450f1f2f
Normal-3466.png,Normal,Normal/images/Normal-3466.png,Normal/masks/Normal-3466.png,True,G12348,train,7f3007fc32e68960aa4bdff4cb1bdd95,d03de6c61e1d4c4d
Normal-3467.png,Normal,Normal/images/Normal-3467.png,Normal/masks/Normal-3467.png,True,G12349,train,a9405c75c5ef479e9ce0c6f021c50295,d42cd3d30b4e8a5b
Normal-3468.png,Normal,Normal/images/Normal-3468.png,Normal/masks/Normal-3468.png,True,G12350,train,31c85043d9701d81a2d75851befb4367,cc3c8bbdb4a492c3
Normal-3469.png,Normal,Normal/images/Normal-3469.png,Normal/masks/Normal-3469.png,True,G12351,train,1cca96853b6087cd4f42f53796aece1e,d927161e431b171f
Normal-347.png,Normal,Normal/images/Normal-347.png,Normal/masks/Normal-347.png,True,G12352,train,97915ac0c1649715675f9ddaf3882d08,d467c74ec149e8e4
Normal-3470.png,Normal,Normal/images/Normal-3470.png,Normal/masks/Normal-3470.png,True,G12353,train,c4ad3d5b999384b3fd4f9d7963d8f1a5,b93986161e4d1f1d
Normal-3471.png,Normal,Normal/images/Normal-3471.png,Normal/masks/Normal-3471.png,True,G12354,train,b1f56b1f43516cf97480eebfd7b295c6,dc3d534b4b4a0b2b
Normal-3472.png,Normal,Normal/images/Normal-3472.png,Normal/masks/Normal-3472.png,True,G12355,val,6e5e6005c28e7a3b96edd384fb555232,d42d438b0c7d6c7c
Normal-3473.png,Normal,Normal/images/Normal-3473.png,Normal/masks/Normal-3473.png,True,G12356,test,8559fe962fae74c61bdc8eb6a9722cdc,896d871a175e38e9
Normal-3474.png,Normal,Normal/images/Normal-3474.png,Normal/masks/Normal-3474.png,True,G12357,val,6294aab1552f5c9dc06aab496d6e2650,c82dc34f0ecf0d0f
Normal-3475.png,Normal,Normal/images/Normal-3475.png,Normal/masks/Normal-3475.png,True,G12358,test,700291e51d89a7857388aa3d3aa321a3,d96d861a9547191f
Normal-3476.png,Normal,Normal/images/Normal-3476.png,Normal/masks/Normal-3476.png,True,G12359,train,ea30d4aa8eb952965edc5018fe56c50a,9979c61ed4461e59
Normal-3477.png,Normal,Normal/images/Normal-3477.png,Normal/masks/Normal-3477.png,True,G12360,train,1cdba0983ca2d50b3e9f5408c1155af9,c96ad21f18473b69
Normal-3478.png,Normal,Normal/images/Normal-3478.png,Normal/masks/Normal-3478.png,True,G12361,val,c7f75934a331e1586e4aa89a15a3155d,8978845a597d7959
Normal-3479.png,Normal,Normal/images/Normal-3479.png,Normal/masks/Normal-3479.png,True,G12362,train,ea877405af1b638f84ffaf2311c50b10,d97f861854591d4e
Normal-348.png,Normal,Normal/images/Normal-348.png,Normal/masks/Normal-348.png,True,G12363,train,b4136fb8bc5ee0c9e474b655e21ab0d4,902fce5e5d4d1d44
Normal-3480.png,Normal,Normal/images/Normal-3480.png,Normal/masks/Normal-3480.png,True,G12364,train,ca37a9a3485e2f6db3a2c03d640061de,d9adc61bc41f184e
Normal-3481.png,Normal,Normal/images/Normal-3481.png,Normal/masks/Normal-3481.png,True,G12365,train,bde9b7354155eee76f183f377bf8f5b4,91701e1a9e7961f9
Normal-3482.png,Normal,Normal/images/Normal-3482.png,Normal/masks/Normal-3482.png,True,G12366,train,72d7803fb28931a076903d3082e36176,dd3ec65b144f1a48
Normal-3483.png,Normal,Normal/images/Normal-3483.png,Normal/masks/Normal-3483.png,True,G12367,train,a20025d9c0d283151a14cd83c08119b4,d02d0b975f3ce087
Normal-3484.png,Normal,Normal/images/Normal-3484.png,Normal/masks/Normal-3484.png,True,G12368,train,770cac1517aa98a305d3e9ea37f36d02,dc67d30f471e4518
Normal-3485.png,Normal,Normal/images/Normal-3485.png,Normal/masks/Normal-3485.png,True,G12369,train,d74970d0c689615f83fec09e41b60f8d,992d065a455af96e
Normal-3486.png,Normal,Normal/images/Normal-3486.png,Normal/masks/Normal-3486.png,True,G12370,train,bb04fed6be5926758b17394587ae8e6b,9978871a19457bbc
Normal-3487.png,Normal,Normal/images/Normal-3487.png,Normal/masks/Normal-3487.png,True,G12371,train,20f703664bc28c861d04f4a9c88b5c04,997d861e18693db8
Normal-3488.png,Normal,Normal/images/Normal-3488.png,Normal/masks/Normal-3488.png,True,G12372,train,09f538a22459b9efcfc5998ca55a86b1,dc3e430f465f184e
Normal-3489.png,Normal,Normal/images/Normal-3489.png,Normal/masks/Normal-3489.png,True,G12373,train,6598d7615b339f9828533528dca53acb,d43cd35b0f422f0b
Normal-349.png,Normal,Normal/images/Normal-349.png,Normal/masks/Normal-349.png,True,G12374,train,3856ba6f71092150ddb3111caa20bb17,903be63a5c4938fc
Normal-3490.png,Normal,Normal/images/Normal-3490.png,Normal/masks/Normal-3490.png,True,G12375,val,295132a49ae6d9b270aaa1253fb752ec,9139d6c71a1d464f
Normal-3491.png,Normal,Normal/images/Normal-3491.png,Normal/masks/Normal-3491.png,True,G12376,val,a43f2f9268a880b9de8f459240e32162,d42f430b2dfcb4d0
Normal-3492.png,Normal,Normal/images/Normal-3492.png,Normal/masks/Normal-3492.png,True,G12377,test,9a306748906cc22126d7e3437351e0c4,d939c61a554f1d49
Normal-3493.png,Normal,Normal/images/Normal-3493.png,Normal/masks/Normal-3493.png,True,G12378,test,9047d6c93108d1a4f2c4f09c861cd6b6,d129065e1a5d5f5c
Normal-3494.png,Normal,Normal/images/Normal-3494.png,Normal/masks/Normal-3494.png,True,G12379,train,fe185b20e99b33ff586b5e87a00398ce,dd28d31b51431f59
Normal-3495.png,Normal,Normal/images/Normal-3495.png,Normal/masks/Normal-3495.png,True,G12380,train,572c3aaf4be4bc2daf40e91265e7cbcd,802ec6971bcdee98
Normal-3496.png,Normal,Normal/images/Normal-3496.png,Normal/masks/Normal-3496.png,True,G12381,train,04f1ef90723726221934ab263cbd7558,d824d60fd90ffc28
Normal-3497.png,Normal,Normal/images/Normal-3497.png,Normal/masks/Normal-3497.png,True,G12382,train,baee8e12aa30ed76ef51a847b8b1e2db,dc06c30f461a4f5f
Normal-3498.png,Normal,Normal/images/Normal-3498.png,Normal/masks/Normal-3498.png,True,G12383,test,b006e3246f420ca5116fb46c7d0ec1e9,c1785a43191fee2e
Normal-3499.png,Normal,Normal/images/Normal-3499.png,Normal/masks/Normal-3499.png,True,G12384,train,ae32e9abf9b87768fd9cff769766eb6b,c03d531f5a0b5f0b
Normal-35.png,Normal,Normal/images/Normal-35.png,Normal/masks/Normal-35.png,True,G12385,val,042a755f2cb676a57a89484b1d991fea,f97bc7965a0c4318
Normal-350.png,Normal,Normal/images/Normal-350.png,Normal/masks/Normal-350.png,True,G12386,train,402a9961092a4ac3bc5b191b4faa6a20,c1eae758e0cd39c4
Normal-3500.png,Normal,Normal/images/Normal-3500.png,Normal/masks/Normal-3500.png,True,G12387,val,0e8ae8119c183a48072a00fdbfdff28b,d42f424f0d2c2f3d
Normal-3501.png,Normal,Normal/images/Normal-3501.png,Normal/masks/Normal-3501.png,True,G12388,train,4aba80c133dfb2ceeadf98d0d66a2b7f,897d861e1d191d4f
Normal-3502.png,Normal,Normal/images/Normal-3502.png,Normal/masks/Normal-3502.png,True,G12389,train,d78a6960b9702183b1ac383edf82ffac,8169961a1d5e39de
Normal-3503.png,Normal,Normal/images/Normal-3503.png,Normal/masks/Normal-3503.png,True,G12390,test,c3d185eaf8c48c294cea2e6616930abb,e979561a192d3f22
Normal-3504.png,Normal,Normal/images/Normal-3504.png,Normal/masks/Normal-3504.png,True,G12391,train,82ef65cf1ade0689941121252f4d8e68,997d871a85cf1a85
Normal-3505.png,Normal,Normal/images/Normal-3505.png,Normal/masks/Normal-3505.png,True,G12392,train,35a5f70ca21ce3a919c92806fb848cda,ec3c525353c7451b
Normal-3506.png,Normal,Normal/images/Normal-3506.png,Normal/masks/Normal-3506.png,True,G12393,train,48a3f186cd25ebc9bf68b66394239c98,dc3dc20b0f4a1f5c
Normal-3507.png,Normal,Normal/images/Normal-3507.png,Normal/masks/Normal-3507.png,True,G12394,train,24cf282afbb834a511e5a147d0374767,886dc6861b495f1f
Normal-3508.png,Normal,Normal/images/Normal-3508.png,Normal/masks/Normal-3508.png,True,G12395,train,a52c55e7565d39d6b331ea6e8e64e6e9,8979871a1f5d584d
Normal-3509.png,Normal,Normal/images/Normal-3509.png,Normal/masks/Normal-3509.png,True,G12396,val,2985119e8549e26534a7f6698b14dea2,9d6c420b4b4d2e7e
Normal-351.png,Normal,Normal/images/Normal-351.png,Normal/masks/Normal-351.png,True,G12397,train,cf55ad0a53cfd485a9197959428432f8,d57f871b898d68a0
Normal-3510.png,Normal,Normal/images/Normal-3510.png,Normal/masks/Normal-3510.png,True,G12398,test,59d467f8deaa39125586b6d0ad7e86b8,d96a021a4f385f2f
Normal-3511.png,Normal,Normal/images/Normal-3511.png,Normal/masks/Normal-3511.png,True,G12399,train,7a1fee8ff8ea02d6683a351991ce1422,997816195f7839a5
Normal-3512.png,Normal,Normal/images/Normal-3512.png,Normal/masks/Normal-3512.png,True,G12400,test,c1549ff6ac74c8bed489361dddcb0fe4,99bc8f29af2b09c2
Normal-3513.png,Normal,Normal/images/Normal-3513.png,Normal/masks/Normal-3513.png,True,G12401,train,693695a84993c48e10a90d8e039c406f,b909061e1c5e5adf
Normal-3514.png,Normal,Normal/images/Normal-3514.png,Normal/masks/Normal-3514.png,True,G12402,train,3974c8ba61fa13a8c75275b53fca2bc9,917846161a5d5f3b
Normal-3515.png,Normal,Normal/images/Normal-3515.png,Normal/masks/Normal-3515.png,True,G12403,train,c835efba7d970d0ad3abf015b92fd172,dd3ac25f0d043d1d
Normal-3516.png,Normal,Normal/images/Normal-3516.png,Normal/masks/Normal-3516.png,True,G12404,train,97fcea52dff8b1b06247f871d88d0ce1,9978c61a1da91bb5
Normal-3517.png,Normal,Normal/images/Normal-3517.png,Normal/masks/Normal-3517.png,True,G12405,train,a011be04c172ae62cd201ed5424528c5,d979c61a154e1bb1
Normal-3518.png,Normal,Normal/images/Normal-3518.png,Normal/masks/Normal-3518.png,True,G12406,train,2e8af1fc1b6b00bb2b50063dbed08477,987d861a196d3ae5
Normal-3519.png,Normal,Normal/images/Normal-3519.png,Normal/masks/Normal-3519.png,True,G12407,train,83fe1344afd5a23ef7fcbf4b2edae49c,917bd61e580b3b19
Normal-352.png,Normal,Normal/images/Normal-352.png,Normal/masks/Normal-352.png,True,G12408,train,7687f813ca926b885d869eb4665206d8,912fcf1b4d493c4c
Normal-3520.png,Normal,Normal/images/Normal-3520.png,Normal/masks/Normal-3520.png,True,G12409,train,bfcd7b222bb8031c38b245222d138a07,d42f071f0d0d1d5d
Normal-3521.png,Normal,Normal/images/Normal-3521.png,Normal/masks/Normal-3521.png,True,G12410,train,b28de453a3c8ae0d68d44392f881ef88,887f071a161f1e4f
Normal-3522.png,Normal,Normal/images/Normal-3522.png,Normal/masks/Normal-3522.png,True,G12411,train,a30557ac00f944e7b9ff4084dc494ced,8138c6571e4e4f3b
Normal-3523.png,Normal,Normal/images/Normal-3523.png,Normal/masks/Normal-3523.png,True,G12412,train,75f8b39030054d0e3e6e4ff9263ffcfc,d918870f1cede538
Normal-3524.png,Normal,Normal/images/Normal-3524.png,Normal/masks/Normal-3524.png,True,G12413,train,635a911b6685108e50edf1b8854613d7,dc3cc2435b424f1f
Normal-3525.png,Normal,Normal/images/Normal-3525.png,Normal/masks/Normal-3525.png,True,G12414,train,a8c45d7d7780fabe06bad676de9f0ed0,9779f884e5e5045c
Normal-3526.png,Normal,Normal/images/Normal-3526.png,Normal/masks/Normal-3526.png,True,G12415,val,a4fb2990af9cfef846e83a17b367f9be,9168165e195f5a79
Normal-3527.png,Normal,Normal/images/Normal-3527.png,Normal/masks/Normal-3527.png,True,G12416,val,d788517bc04513cc6f60e0fe43fd8678,d53f03131f4d1a4d
Normal-3528.png,Normal,Normal/images/Normal-3528.png,Normal/masks/Normal-3528.png,True,G12417,val,d569b2b91fbb66cb7bbc2df94bcad4e2,d87dc6531d485b19
Normal-3529.png,Normal,Normal/images/Normal-3529.png,Normal/masks/Normal-3529.png,True,G12418,test,63c5e3092302782cc68d720bc94179f7,d568065e194f1d79
Normal-353.png,Normal,Normal/images/Normal-353.png,Normal/masks/Normal-353.png,True,G12419,test,e803577736ebc4562c9d5ea0e9399fcd,98e4a75bad78ac0c
Normal-3530.png,Normal,Normal/images/Normal-3530.png,Normal/masks/Normal-3530.png,True,G12420,test,f51538b80dc067eec3a0bd05a8ffe221,dc3cd30b444b1f4e
Normal-3531.png,Normal,Normal/images/Normal-3531.png,Normal/masks/Normal-3531.png,True,G12421,train,7ffdc5eba7118f619b621ca68d976cb3,d979441e1a4f1b39
Normal-3532.png,Normal,Normal/images/Normal-3532.png,Normal/masks/Normal-3532.png,True,G12422,train,329ac1e513d188524c684760db5e2c31,d13956471b1c2f39
Normal-3533.png,Normal,Normal/images/Normal-3533.png,Normal/masks/Normal-3533.png,True,G12423,train,de1298fa31257d1b0d5af9353e1ace7d,dc2a435d4a2f0c6f
Normal-3534.png,Normal,Normal/images/Normal-3534.png,Normal/masks/Normal-3534.png,True,G12424,train,55f4a94622a6775bc99047f76476b5cd,9168061e1a5b3f7b
Normal-3535.png,Normal,Normal/images/Normal-3535.png,Normal/masks/Normal-3535.png,True,G12425,train,cf4a07d90d702ef954258b3872aa68d2,c87b171a1f0d3a39
Normal-3536.png,Normal,Normal/images/Normal-3536.png,Normal/masks/Normal-3536.png,True,G12426,train,924c46c4af5fc9d775560b1a067d1b1d,dc42c30b4c2acdbf
Normal-3537.png,Normal,Normal/images/Normal-3537.png,Normal/masks/Normal-3537.png,True,G12427,train,099598c52b5732e51239b90b733dbb24,dd2dd50a1e4d0a3d
Normal-3538.png,Normal,Normal/images/Normal-3538.png,Normal/masks/Normal-3538.png,True,G12428,val,11b0571b63b002b5c0c7f3b93f8f7c8d,91694e16595ece78
Normal-3539.png,Normal,Normal/images/Normal-3539.png,Normal/masks/Normal-3539.png,True,G12429,val,8e6727a4b7579928383e4a3fb202119a,985f061a185b3f3e
Normal-354.png,Normal,Normal/images/Normal-354.png,Normal/masks/Normal-354.png,True,G12430,test,bc2885e8e1406e35d0f8d3cf3a39b0fa,917cc71acd4f2994
Normal-3540.png,Normal,Normal/images/Normal-3540.png,Normal/masks/Normal-3540.png,True,G12431,train,7e7091c3b73fc39d208778f04592c70b,c97956021f1d2b3b
Normal-3541.png,Normal,Normal/images/Normal-3541.png,Normal/masks/Normal-3541.png,True,G12432,train,6b396ca9a0d412441ce738b2ee66cccc,9d6b071e5a493c78
Normal-3542.png,Normal,Normal/images/Normal-3542.png,Normal/masks/Normal-3542.png,True,G12433,train,3043c970678a0155c377911ec9decf49,d13b55171e1e1969
Normal-3543.png,Normal,Normal/images/Normal-3543.png,Normal/masks/Normal-3543.png,True,G12434,train,5779374db5a78b576163776c8dac6ff5,9960861b1f5e397a
Normal-3544.png,Normal,Normal/images/Normal-3544.png,Normal/masks/Normal-3544.png,True,G12435,train,19d105f6287f60d432e260bab4403dd2,d92b521f0d893da6
Normal-3545.png,Normal,Normal/images/Normal-3545.png,Normal/masks/Normal-3545.png,True,G12436,train,b44ec0c3712ae00bc2c033bce76b446c,912c07191f3f3b2b
Normal-3546.png,Normal,Normal/images/Normal-3546.png,Normal/masks/Normal-3546.png,True,G12437,train,9e2d7c78f1b035b5eb28e7866854cbe5,9c7cc35b090e3d99
Normal-3547.png,Normal,Normal/images/Normal-3547.png,Normal/masks/Normal-3547.png,True,G12438,val,bc73692c778b662be18aad94a3855fb4,897a07061a5f6a7e
Normal-3548.png,Normal,Normal/images/Normal-3548.png,Normal/masks/Normal-3548.png,True,G12439,test,9106cdf534e69c95f675ba291bb3f963,c939c61e1a466f1b
Normal-3549.png,Normal,Normal/images/Normal-3549.png,Normal/masks/Normal-3549.png,True,G12440,test,218664e524a6c211bf31583b464df746,dd28561f582938f5
Normal-355.png,Normal,Normal/images/Normal-355.png,Normal/masks/Normal-355.png,True,G12441,train,31905d0bab3b868b4481c9835547b6ec,957e271b49147a96
Normal-3550.png,Normal,Normal/images/Normal-3550.png,Normal/masks/Normal-3550.png,True,G12442,train,1cd6d01a13f890558b76ca8517285160,8969065f39ececa1
Normal-3551.png,Normal,Normal/images/Normal-3551.png,Normal/masks/Normal-3551.png,True,G12443,train,26a59e82f273aa37df5ff7f0b3f93394,8974435b4b6d3b29
Normal-3552.png,Normal,Normal/images/Normal-3552.png,Normal/masks/Normal-3552.png,True,G12444,train,135133b35b1cb80dab68021859a6b1ee,9138ef06be018e7e
Normal-3553.png,Normal,Normal/images/Normal-3553.png,Normal/masks/Normal-3553.png,True,G12445,train,abbf492f27051f150690de4752384815,857887c69a89875f
Normal-3554.png,Normal,Normal/images/Normal-3554.png,Normal/masks/Normal-3554.png,True,G12446,train,7548d78cfa2289072604a398568f2688,ad2cd60bd5461d56
Normal-3555.png,Normal,Normal/images/Normal-3555.png,Normal/masks/Normal-3555.png,True,G12447,train,d92348db0aab58c000f44333ed15e953,d42cc30b0d2e3d7d
Normal-3556.png,Normal,Normal/images/Normal-3556.png,Normal/masks/Normal-3556.png,True,G12448,train,307c9a4c950f4bb76ff012e985ebe85b,807b87061e5a4f5f
Normal-3557.png,Normal,Normal/images/Normal-3557.png,Normal/masks/Normal-3557.png,True,G12449,train,67ba3536d373f7cb8e158f14bfbc4c4c,816896c71b1ece4f
Normal-3558.png,Normal,Normal/images/Normal-3558.png,Normal/masks/Normal-3558.png,True,G12450,train,fdd9e552c8b0638eeb28f2aeaa25b449,cc3d421f524f1a5d
Normal-3559.png,Normal,Normal/images/Normal-3559.png,Normal/masks/Normal-3559.png,True,G12451,train,3ef6fb69470039ae713d975758a2a400,cd20b24fce1bcc1b
Normal-356.png,Normal,Normal/images/Normal-356.png,Normal/masks/Normal-356.png,True,G12452,train,8235c7a2f24c8d05bba371f3eb74e097,946fcf1e8d4c0cc6
Normal-3560.png,Normal,Normal/images/Normal-3560.png,Normal/masks/Normal-3560.png,True,G12453,train,8cf3a9beada8430692da6566eebbd7d3,996916184d1b5dcf
Normal-3561.png,Normal,Normal/images/Normal-3561.png,Normal/masks/Normal-3561.png,True,G12454,val,653913d98a0f17a1a130d04fa371796b,997f465a095b590b
Normal-3562.png,Normal,Normal/images/Normal-3562.png,Normal/masks/Normal-3562.png,True,G12455,val,2b8e7f1b600f084d4250da0bbd6d5014,d5a8475a074e5a5e
Normal-3563.png,Normal,Normal/images/Normal-3563.png,Normal/masks/Normal-3563.png,True,G12456,test,814041b4a4c89d16667fc9b13beb62ce,8c6d430bd94f1a9d
Normal-3564.png,Normal,Normal/images/Normal-3564.png,Normal/masks/Normal-3564.png,True,G12457,val,b1e611c28a32466415d4498df7739be5,992cc69b8b8b89ad
Normal-3565.png,Normal,Normal/images/Normal-3565.png,Normal/masks/Normal-3565.png,True,G12458,test,683e67ccddd3e1f6d89d49cffa139b03,b92f861a0f4e1b4d
Normal-3566.png,Normal,Normal/images/Normal-3566.png,Normal/masks/Normal-3566.png,True,G12459,train,f2f4ff2dd6cc46e5065dce768e410ca0,996dc71a445a39cd
Normal-3567.png,Normal,Normal/images/Normal-3567.png,Normal/masks/Normal-3567.png,True,G12460,train,83622531e5ab553b08b5c9e5b6cc095b,896d865e5b4d585a
Normal-3568.png,Normal,Normal/images/Normal-3568.png,Normal/masks/Normal-3568.png,True,G12461,train,212c97c357e19fe91be5ceea4d5ae92b,f91d865a186939f8
Normal-3569.png,Normal,Normal/images/Normal-3569.png,Normal/masks/Normal-3569.png,True,G12462,test,bcd504b6dbbd1c7dc93e7c1ac64d116d,8169069e590b7d7b
Normal-357.png,Normal,Normal/images/Normal-357.png,Normal/masks/Normal-357.png,True,G12463,train,e477d8ef6971ca4f2c9d578e8068d998,9979833a4d3f5905
Normal-3570.png,Normal,Normal/images/Normal-3570.png,Normal/masks/Normal-3570.png,True,G12464,train,6eab07335b8994575c2c1df94d89f34d,817f47521a173a1f
Normal-3571.png,Normal,Normal/images/Normal-3571.png,Normal/masks/Normal-3571.png,True,G12465,train,ea0cfedb51732d5d70819d4a25b4d238,d83d43191b6c79b1
Normal-3572.png,Normal,Normal/images/Normal-3572.png,Normal/masks/Normal-3572.png,True,G12466,train,956482c0256006ff03514aa7a914da63,cc2c425a0decbcfc
Normal-3573.png,Normal,Normal/images/Normal-3573.png,Normal/masks/Normal-3573.png,True,G12467,train,4c6588fac1bb93571b14c07fb2664ec3,990d061e1d5e595f
Normal-3574.png,Normal,Normal/images/Normal-3574.png,Normal/masks/Normal-3574.png,True,G12468,train,965d68ad5a4389654fb2810731ce9f66,d92bd7161a1d4a39
Normal-3575.png,Normal,Normal/images/Normal-3575.png,Normal/masks/Normal-3575.png,True,G12469,train,97d56c9dc6fae643b589f525f3485bb5,9968475a574b194f
Normal-3576.png,Normal,Normal/images/Normal-3576.png,Normal/masks/Normal-3576.png,True,G12470,test,6b9ccd0f9838753572f16481c644f597,d87fc65a1c487919
Normal-3577.png,Normal,Normal/images/Normal-3577.png,Normal/masks/Normal-3577.png,True,G12471,val,ec0ff4e968427a07c47806286729cf4a,886c175b455b7c6c
Normal-3578.png,Normal,Normal/images/Normal-3578.png,Normal/masks/Normal-3578.png,True,G12472,test,fddc4e6c76b7501f1e2bc28740d09e40,dc3cc21b454f1d4b
Normal-3579.png,Normal,Normal/images/Normal-3579.png,Normal/masks/Normal-3579.png,True,G12473,test,b836f48c68421ee1c77b1ed30feaf685,d979871e184f1e19
Normal-358.png,Normal,Normal/images/Normal-358.png,Normal/masks/Normal-358.png,True,G12474,train,7dbf8d5d5822cba4da7b8221a8deaf6e,913bd61a9c7939b0
Normal-3580.png,Normal,Normal/images/Normal-3580.png,Normal/masks/Normal-3580.png,True,G12475,train,9da7775d495ba85df74417e3e7b79b72,813957531b1eeb19
Normal-3581.png,Normal,Normal/images/Normal-3581.png,Normal/masks/Normal-3581.png,True,G12476,train,8644a5497eb545f9f8f8b87f6c1e61db,897846867aa57eb4
Normal-3582.png,Normal,Normal/images/Normal-3582.png,Normal/masks/Normal-3582.png,True,G12477,train,7dace52284797bcf55d157208916f104,cd6882570f5c0f5d
Normal-3583.png,Normal,Normal/images/Normal-3583.png,Normal/masks/Normal-3583.png,True,G12478,train,5d6e576a55ac926ae6b142c92b9af972,d939460b19856fb9
Normal-3584.png,Normal,Normal/images/Normal-3584.png,Normal/masks/Normal-3584.png,True,G12479,val,23b540f89d3a8bdc21ad45a0545c9501,9917861a5e7a6a6a
Normal-3585.png,Normal,Normal/images/Normal-3585.png,Normal/masks/Normal-3585.png,True,G12480,val,59fd2eb25e61300da2132c582ca0bcbc,91698e1e1a497d79
Normal-3586.png,Normal,Normal/images/Normal-3586.png,Normal/masks/Normal-3586.png,True,G12481,train,f158eae75766b617d0962fd0bb6a3272,8d7dd11352071f5a
Normal-3587.png,Normal,Normal/images/Normal-3587.png,Normal/masks/Normal-3587.png,True,G12482,train,1ecc44faf704200f318fc444720eb383,913f465a184b3b3d
Normal-3588.png,Normal,Normal/images/Normal-3588.png,Normal/masks/Normal-3588.png,True,G12483,train,d75c3060107e6eb66c715db42ff55a5f,997bc7861e195a1a
Normal-3589.png,Normal,Normal/images/Normal-3589.png,Normal/masks/Normal-3589.png,True,G12484,train,9e201b9d646da73dad610d8ac6ccfd58,817f061e1f5e187a
Normal-359.png,Normal,Normal/images/Normal-359.png,Normal/masks/Normal-359.png,True,G12485,train,a42e4c3f3a69918aa6f30db2449ae57b,9129e23ab86939f9
Normal-3590.png,Normal,Normal/images/Normal-3590.png,Normal/masks/Normal-3590.png,True,G12486,train,711a0b857ea6538e68f2af79a3d50732,dd2c06821b554f5f
Normal-3591.png,Normal,Normal/images/Normal-3591.png,Normal/masks/Normal-3591.png,True,G12487,train,dcebba4ddd6061177029adbbe618c211,d1b938da39d964a4
Normal-3592.png,Normal,Normal/images/Normal-3592.png,Normal/masks/Normal-3592.png,True,G12488,train,d6d4f63cdc95e4fd118caf6fb5f35e32,99798712184f787b
Normal-3593.png,Normal,Normal/images/Normal-3593.png,Normal/masks/Normal-3593.png,True,G12489,train,b5f572d35750343a3ba3b068453a495a,cd4e474b4d2e2ce4
Normal-3594.png,Normal,Normal/images/Normal-3594.png,Normal/masks/Normal-3594.png,True,G12490,train,c2a6d48b5e1f20d894b407843c87fa79,892e861f6c7c68ec
Normal-3595.png,Normal,Normal/images/Normal-3595.png,Normal/masks/Normal-3595.png,True,G12491,train,86e76fa076e78ab095a48bacd39a8a76,8879971e4c5f1a69
Normal-3596.png,Normal,Normal/images/Normal-3596.png,Normal/masks/Normal-3596.png,True,G12492,train,f2807b2655d5a2a2423e19af259b2bb0,c83d47431f464b1f
Normal-3597.png,Normal,Normal/images/Normal-3597.png,Normal/masks/Normal-3597.png,True,G12493,train,f66134b0e2bbc14a53666691e3979d46,9129065e1e5a5f7a
Normal-3598.png,Normal,Normal/images/Normal-3598.png,Normal/masks/Normal-3598.png,True,G12494,train,ec8f1bf6c67e86b85776f70db20f6173,dd2dd01a1f4c0a9f
Normal-3599.png,Normal,Normal/images/Normal-3599.png,Normal/masks/Normal-3599.png,True,G12495,val,f821b91c5a11ee46e779d8a596d3f252,dc2fc2590f422f1d
Normal-36.png,Normal,Normal/images/Normal-36.png,Normal/masks/Normal-36.png,True,G12496,test,12c92ff19b1ec99a6d6d9ac49ead0e42,d13d8613190f2e3f
Normal-360.png,Normal,Normal/images/Normal-360.png,Normal/masks/Normal-360.png,True,G12497,val,a9ea5f721b38d7c76df5c6824521d9b9,906ac35f4d4d5d4c
Normal-3600.png,Normal,Normal/images/Normal-3600.png,Normal/masks/Normal-3600.png,True,G12498,test,d8233f6fde7981b54eca352ec66aec57,d939c61a19693f68
Normal-3601.png,Normal,Normal/images/Normal-3601.png,Normal/masks/Normal-3601.png,True,G12499,val,a7f4f641c34979aea346a064c9a662d5,d87d425f4a4d4a5c
Normal-3602.png,Normal,Normal/images/Normal-3602.png,Normal/masks/Normal-3602.png,True,G12500,test,f32a520403e8828ae800848ea0d1cb4f,d42d4a5f1d406d3d
Normal-3603.png,Normal,Normal/images/Normal-3603.png,Normal/masks/Normal-3603.png,True,G12501,train,6942f1edf48691bccebea2cde8f14d5d,d97d061a4c1f59c5
Normal-3604.png,Normal,Normal/images/Normal-3604.png,Normal/masks/Normal-3604.png,True,G12502,train,3f020edae8f6e0e66280650ae7a97292,9929861e1d5f1a5d
Normal-3605.png,Normal,Normal/images/Normal-3605.png,Normal/masks/Normal-3605.png,True,G12503,train,98a2bc7569de65424facb51a8419246a,9979c65619495b1b
Normal-3606.png,Normal,Normal/images/Normal-3606.png,Normal/masks/Normal-3606.png,True,G12504,train,21db12fdba9699ea037aceead619a82e,813e06825b5dfe78
Normal-3607.png,Normal,Normal/images/Normal-3607.png,Normal/masks/Normal-3607.png,True,G12505,train,ac3f424ea72805c08b5dcc4b9b4bf5d0,dd28c21f4e5e584b
Normal-3608.png,Normal,Normal/images/Normal-3608.png,Normal/masks/Normal-3608.png,True,G12506,train,4f4748b10160e29655cc38602f2ffd82,997fc41a19491bad
Normal-3609.png,Normal,Normal/images/Normal-3609.png,Normal/masks/Normal-3609.png,True,G12507,train,bed09ec4cd529dc0b7793d840d6aef5d,ce3d410b0fac2e3e
Normal-361.png,Normal,Normal/images/Normal-361.png,Normal/masks/Normal-361.png,True,G12508,train,aa0bc40efbb15510602193cbd17e479e,d56f871b8d0d0cc6
Normal-3610.png,Normal,Normal/images/Normal-3610.png,Normal/masks/Normal-3610.png,True,G12509,train,ece5c53d7773c93ce27e46d85917f6f0,d42cd31f544f3c46
Normal-3611.png,Normal,Normal/images/Normal-3611.png,Normal/masks/Normal-3611.png,True,G12510,train,098d2d819b29fe6e6e6c2e476bd213c3,dc2ac30f3c22353f
Normal-3612.png,Normal,Normal/images/Normal-3612.png,Normal/masks/Normal-3612.png,True,G12511,train,25580b618d428fa300dd7213ca23039c,dd3c865b0d0a39e6
Normal-3613.png,Normal,Normal/images/Normal-3613.png,Normal/masks/Normal-3613.png,True,G12512,train,7e3b728c9263c9df384953f6ede064e6,dd2cc20b0d8e1baf
Normal-3614.png,Normal,Normal/images/Normal-3614.png,Normal/masks/Normal-3614.png,True,G12513,train,86a2eabbedbb141e964bacf26560e225,de2fd2410d6e2d1c
Normal-3615.png,Normal,Normal/images/Normal-3615.png,Normal/masks/Normal-3615.png,True,G12514,train,be75af0f38746464cc5a262e14ad4039,9979871618cc7e78
Normal-3616.png,Normal,Normal/images/Normal-3616.png,Normal/masks/Normal-3616.png,True,G12515,train,214b07c0b8c02add703085a78f0ba49b,cc3dc30f100f2fbc
Normal-3617.png,Normal,Normal/images/Normal-3617.png,Normal/masks/Normal-3617.png,True,G12516,val,4ec07eed69c33528132a45921b8a8a22,897f861a596c38e9
Normal-3618.png,Normal,Normal/images/Normal-3618.png,Normal/masks/Normal-3618.png,True,G12517,val,b9f4e7855d132d128740f61089040834,882d571f1f4d6c38
Normal-3619.png,Normal,Normal/images/Normal-3619.png,Normal/masks/Normal-3619.png,True,G12518,test,92d6351f2b3eef9fe31ce8dc737add40,d939d20f1ad50a3d
Normal-362.png,Normal,Normal/images/Normal-362.png,Normal/masks/Normal-362.png,True,G12519,val,f75ad155df2d585204e7de097f6067e9,c16e8741c57c68de
Normal-3620.png,Normal,Normal/images/Normal-3620.png,Normal/masks/Normal-3620.png,True,G12520,test,b165a574c7812fb4c7e2cf27834642c1,dc6d530f4e0d1c2d
Normal-3621.png,Normal,Normal/images/Normal-3621.png,Normal/masks/Normal-3621.png,True,G12521,test,74e7d7ec6fbace426839dd36fe9072ae,8178769678915e9e
Normal-3622.png,Normal,Normal/images/Normal-3622.png,Normal/masks/Normal-3622.png,True,G12522,train,badfd8a66236d51a683704d637ab43c8,8168861e18df7a7b
Normal-3623.png,Normal,Normal/images/Normal-3623.png,Normal/masks/Normal-3623.png,True,G12523,train,42cb5eefc27ae33d054f81f11a2bc7ce,c3352a3db6d0c94e
Normal-3624.png,Normal,Normal/images/Normal-3624.png,Normal/masks/Normal-3624.png,True,G12524,train,c50ce81d2d6d47b33bf44bb344074c6c,8168c65e574e1a5f
Normal-3625.png,Normal,Normal/images/Normal-3625.png,Normal/masks/Normal-3625.png,True,G12525,train,811cdee6b1f08fbf57d7bdb01799404e,d92dc61f844e1e4b
Normal-3626.png,Normal,Normal/images/Normal-3626.png,Normal/masks/Normal-3626.png,True,G12526,train,8a762c64405ece808158b95107ea2c4d,f96d461b441b594b
Normal-3627.png,Normal,Normal/images/Normal-3627.png,Normal/masks/Normal-3627.png,True,G12527,train,95a13258f26596362444d9427e8c2173,9c42934b4d38ed3e
Normal-3628.png,Normal,Normal/images/Normal-3628.png,Normal/masks/Normal-3628.png,True,G12528,train,e921413fe30f6dfb141c1d2491b1e369,9979c65e50c65b58
Normal-3629.png,Normal,Normal/images/Normal-3629.png,Normal/masks/Normal-3629.png,True,G12529,train,374fb4f179fd560e16e47f3a57002352,d42f02494f4f6f4c
Normal-363.png,Normal,Normal/images/Normal-363.png,Normal/masks/Normal-363.png,True,G12530,train,c04be080ae2bb17699833602c6c59bc6,812fc33ae8796969
Normal-3630.png,Normal,Normal/images/Normal-3630.png,Normal/masks/Normal-3630.png,True,G12531,train,61893ace62a7796cc144526dac00ea86,dd2dd0531bd54319
Normal-3631.png,Normal,Normal/images/Normal-3631.png,Normal/masks/Normal-3631.png,True,G12532,train,d6ca8ac15d1adc0e6f533e42223ce37e,882957571f4d3d98
Normal-3632.png,Normal,Normal/images/Normal-3632.png,Normal/masks/Normal-3632.png,True,G12533,train,52f1dad8e4e5946e8c60263f39dfee27,93695c1c1b7ad869
Normal-3633.png,Normal,Normal/images/Normal-3633.png,Normal/masks/Normal-3633.png,True,G12534,train,824669d3265409eecc2299b389cf8422,8929161e5e5e7a78
Normal-3634.png,Normal,Normal/images/Normal-3634.png,Normal/masks/Normal-3634.png,True,G12535,train,c66f8e38afeb6ca62fb86c672c7621f6,dd2d42cf0a54eb1a
Normal-3635.png,Normal,Normal/images/Normal-3635.png,Normal/masks/Normal-3635.png,True,G12536,test,cc010d53b4506c410198eff6258f8b40,dc2e431f0cac3b3c
Normal-3636.png,Normal,Normal/images/Normal-3636.png,Normal/masks/Normal-3636.png,True,G12537,train,8066c18bb583b0a293bc1612819d0044,997dc65a1b471e18
Normal-3637.png,Normal,Normal/images/Normal-3637.png,Normal/masks/Normal-3637.png,True,G12538,val,3c132e74c44f0a9a1b6ee7bcaf699cf2,c93dc65f19442f34
Normal-3638.png,Normal,Normal/images/Normal-3638.png,Normal/masks/Normal-3638.png,True,G12539,val,a37186c9f9a3fa36a14ce52e0412c0ee,d97b451e1a095b39
Normal-3639.png,Normal,Normal/images/Normal-3639.png,Normal/masks/Normal-3639.png,True,G12540,val,439b36ee88ec01f76b2ace0cf83622a8,9979869e1e877a90
Normal-364.png,Normal,Normal/images/Normal-364.png,Normal/masks/Normal-364.png,True,G12541,test,f8f460085a86e02234feb46de8f0139e,8174a77ae9a568e4
Normal-3640.png,Normal,Normal/images/Normal-3640.png,Normal/masks/Normal-3640.png,True,G12542,test,28cb2bc9d9a2be8d72796c242fde920e,dd2e065c055ccd5e
Normal-3641.png,Normal,Normal/images/Normal-3641.png,Normal/masks/Normal-3641.png,True,G12543,train,9518074a7e3c6bcef413e5a74754eaa1,917b431656431f1f
Normal-3642.png,Normal,Normal/images/Normal-3642.png,Normal/masks/Normal-3642.png,True,G12544,train,536726113e3e1ffc6f62beb4aa0b82d3,dc3ac30f0d2c2ead
Normal-3643.png,Normal,Normal/images/Normal-3643.png,Normal/masks/Normal-3643.png,True,G12545,train,92f06843e9fb3866436abc4382f234f8,c82e46571a4f6f38
Normal-3644.png,Normal,Normal/images/Normal-3644.png,Normal/masks/Normal-3644.png,True,G12546,train,a943e9c41d563df6fcba40df04faa176,d96cc61a0c4f38ed
Normal-3645.png,Normal,Normal/images/Normal-3645.png,Normal/masks/Normal-3645.png,True,G12547,train,823676b0c5b88cb29dce4efac5452eeb,dc3dc20f1a8b0fb8
Normal-3646.png,Normal,Normal/images/Normal-3646.png,Normal/masks/Normal-3646.png,True,G12548,train,c853a3d632b76eaeaa1051e8dd4eeb96,d43e42c353554f0f
Normal-3647.png,Normal,Normal/images/Normal-3647.png,Normal/masks/Normal-3647.png,True,G12549,train,996a814479ee3587a4f1d2ca386edb6d,9d3c075a1b4f1c3a
Normal-3648.png,Normal,Normal/images/Normal-3648.png,Normal/masks/Normal-3648.png,True,G12550,train,e6dcb0444d040ec38360bca89ed790b1,dd3c061b4d4e78e4
Normal-3649.png,Normal,Normal/images/Normal-3649.png,Normal/masks/Normal-3649.png,True,G12551,test,8e65b461e4bfa7308c2f22494e3a2277,9928165a5d784bf9
Normal-365.png,Normal,Normal/images/Normal-365.png,Normal/masks/Normal-365.png,True,G12552,train,d95279010611b63182d71a7efc359ff8,917da73a9a0b199c
Normal-3650.png,Normal,Normal/images/Normal-3650.png,Normal/masks/Normal-3650.png,True,G12553,train,d7970e1920e7091aa5c6a4286fe6dd6e,916b1c525a59795b
Normal-3651.png,Normal,Normal/images/Normal-3651.png,Normal/masks/Normal-3651.png,True,G12554,train,be182661770f0598ffe0e8dd34af5e17,cdbec2471a844f1b
Normal-3652.png,Normal,Normal/images/Normal-3652.png,Normal/masks/Normal-3652.png,True,G12555,val,be77e11803609c9df2393a752204d4e2,dc2ed0094f6c1f4e
Normal-3653.png,Normal,Normal/images/Normal-3653.png,Normal/masks/Normal-3653.png,True,G12556,train,1d8b9e6d9442f1f06db954b50f7e24e5,d43f834b57470f0a
Normal-3654.png,Normal,Normal/images/Normal-3654.png,Normal/masks/Normal-3654.png,True,G12557,train,f18a563d5f41f2ccff1944cb9100bacb,dd2e46c31b1d4e0e
Normal-3655.png,Normal,Normal/images/Normal-3655.png,Normal/masks/Normal-3655.png,True,G12558,train,533e46647cb1cc9e9acffff97bb3f861,95694e961a9f689a
Normal-3656.png,Normal,Normal/images/Normal-3656.png,Normal/masks/Normal-3656.png,True,G12559,train,2e71617610b4b0fbb8eef132c1b8ee40,f979c61a544b1d58
Normal-3657.png,Normal,Normal/images/Normal-3657.png,Normal/masks/Normal-3657.png,True,G12560,test,7c24d620b77b3226656bf5dae8904275,81b5521f564e5e1b
Normal-3658.png,Normal,Normal/images/Normal-3658.png,Normal/masks/Normal-3658.png,True,G12561,val,045cc9fd115197e145817cba22322118,c03d43474d1e6f2d
Normal-3659.png,Normal,Normal/images/Normal-3659.png,Normal/masks/Normal-3659.png,True,G12562,val,af9579b8c381d86ba73f70b002fc4af6,dd2e425f1a4c6c78
Normal-366.png,Normal,Normal/images/Normal-366.png,Normal/masks/Normal-366.png,True,G12563,test,acf60475c5d1db84655b7e9b866ae3c4,917fc71e8a4b3941
Normal-3660.png,Normal,Normal/images/Normal-3660.png,Normal/masks/Normal-3660.png,True,G12564,train,de245d687a50580967f66eff8ae854d0,dd2a825fc25a194f
Normal-3661.png,Normal,Normal/images/Normal-3661.png,Normal/masks/Normal-3661.png,True,G12565,train,24676ea0652dcb5a6057d0959c64aa5e,d97ec61b871a190e
Normal-3662.png,Normal,Normal/images/Normal-3662.png,Normal/masks/Normal-3662.png,True,G12566,train,db3bbf8b1d3ad3a8c91a8ee1d008b289,817d041a5f7d7958
Normal-3663.png,Normal,Normal/images/Normal-3663.png,Normal/masks/Normal-3663.png,True,G12567,train,849749e804676465a2a892057b843789,987f841707875b1b
Normal-3664.png,Normal,Normal/images/Normal-3664.png,Normal/masks/Normal-3664.png,True,G12568,train,99df706e9873b8bffe22eaba8ffdbea0,a97f061a1b4d5a1d
Normal-3665.png,Normal,Normal/images/Normal-3665.png,Normal/masks/Normal-3665.png,True,G12569,train,45915a184661a8a0ea8e677e3ccc700c,9179861e19b96a3b
Normal-3666.png,Normal,Normal/images/Normal-3666.png,Normal/masks/Normal-3666.png,True,G12570,test,4e2ffc5add9a0025d61affb6ca46a426,f91d87161d051d1f
Normal-3667.png,Normal,Normal/images/Normal-3667.png,Normal/masks/Normal-3667.png,True,G12571,train,90fcc720343c5b6d36b4d9c7bfea3db4,d939c65a1d481b3d
Normal-3668.png,Normal,Normal/images/Normal-3668.png,Normal/masks/Normal-3668.png,True,G12572,train,4991862e65aaba296a5bcd91cfbd56d1,dc2cd2534f0ad34b
Normal-3669.png,Normal,Normal/images/Normal-3669.png,Normal/masks/Normal-3669.png,True,G12573,train,359897434786a73ef4911d5b95b22133,dc2ec309442e3df6
Normal-367.png,Normal,Normal/images/Normal-367.png,Normal/masks/Normal-367.png,True,G12574,train,040951cd1f5f8e5ef1aed33056428c7a,d025c61fc16d3cc7
Normal-3670.png,Normal,Normal/images/Normal-3670.png,Normal/masks/Normal-3670.png,True,G12575,train,0f87d9bfd4b597201dd641ceb81f450b,816816561b5efa7a
Normal-3671.png,Normal,Normal/images/Normal-3671.png,Normal/masks/Normal-3671.png,True,G12576,train,2891fb1b766057682dc82282912d0591,dd2c5203534f1b2f
Normal-3672.png,Normal,Normal/images/Normal-3672.png,Normal/masks/Normal-3672.png,True,G12577,train,8145bd812313c955bcf6e9cf0756a04b,817816165bf8f8f1
Normal-3673.png,Normal,Normal/images/Normal-3673.png,Normal/masks/Normal-3673.png,True,G12578,val,c2b3da9291bc1c38b6c4c17eda0e8caa,d05b535a5a4d0f1b
Normal-3674.png,Normal,Normal/images/Normal-3674.png,Normal/masks/Normal-3674.png,True,G12579,val,adad32e05bb96cfcf334b4fd6ce60d8e,d6640e0f6d6cec64
Normal-3675.png,Normal,Normal/images/Normal-3675.png,Normal/masks/Normal-3675.png,True,G12580,val,d7c0fd0cbce3294bad31153ca446feee,993816dab8e4b6a5
Normal-3676.png,Normal,Normal/images/Normal-3676.png,Normal/masks/Normal-3676.png,True,G12581,test,1ca0df00480c2f24f826eb64a40a78b8,f879c71e1a4d1919
Normal-3677.png,Normal,Normal/images/Normal-3677.png,Normal/masks/Normal-3677.png,True,G12582,test,f7fecfbca71a192459db1d3696367734,8868c21b0f1b3dbf
Normal-3678.png,Normal,Normal/images/Normal-3678.png,Normal/masks/Normal-3678.png,True,G12583,train,e65c6b0cc8bf484b247f8711d77e9462,dc2ed30b4c4a1eec
Normal-3679.png,Normal,Normal/images/Normal-3679.png,Normal/masks/Normal-3679.png,True,G12584,train,9d6ef75177fc50bf34129a442a4cfd2d,cc2d434b4d4e2e6b
Normal-368.png,Normal,Normal/images/Normal-368.png,Normal/masks/Normal-368.png,True,G12585,train,18a947d05b5692c9f81db8e742fa64c8,903ad65a9c593da5
Normal-3680.png,Normal,Normal/images/Normal-3680.png,Normal/masks/Normal-3680.png,True,G12586,train,7cc7920e2c01212a7d536de849dbcd41,cd74525f451b092f
Normal-3681.png,Normal,Normal/images/Normal-3681.png,Normal/masks/Normal-3681.png,True,G12587,train,1c094d7c1932241933d535c5162933ff,dc2c4219442ebdbf
Normal-3682.png,Normal,Normal/images/Normal-3682.png,Normal/masks/Normal-3682.png,True,G12588,train,e142b1b658ec7bbc20ce29d4f81594e3,9979061a596d3be1
Normal-3683.png,Normal,Normal/images/Normal-3683.png,Normal/masks/Normal-3683.png,True,G12589,train,aa55a89bbc006bfd92c031cf60a993e5,ce3cc34b0eec0d0f
Normal-3684.png,Normal,Normal/images/Normal-3684.png,Normal/masks/Normal-3684.png,True,G12590,train,76214f5942574428ba7ce2b8ad1e218b,992e061b0f5c7c7c
Normal-3685.png,Normal,Normal/images/Normal-3685.png,Normal/masks/Normal-3685.png,True,G12591,train,f2aa4abcb1ee8a36f0dba489ffa00755,893c861a174e1bfd
Normal-3686.png,Normal,Normal/images/Normal-3686.png,Normal/masks/Normal-3686.png,True,G12592,train,98a63ffe248233526dca74c465765c29,dd2fc25b1d431b18
Normal-3687.png,Normal,Normal/images/Normal-3687.png,Normal/masks/Normal-3687.png,True,G12593,test,8edabdb5f428a6983b9392a5cf6b8c38,93691e5e385ac578
Normal-3688.png,Normal,Normal/images/Normal-3688.png,Normal/masks/Normal-3688.png,True,G12594,test,5376be49ff5d6478cdf05242b977ed15,81e84611175e5f5f
Normal-3689.png,Normal,Normal/images/Normal-3689.png,Normal/masks/Normal-3689.png,True,G12595,train,41280b2ed136462a83923b14339d1279,997e06d61a184d7d
Normal-369.png,Normal,Normal/images/Normal-369.png,Normal/masks/Normal-369.png,True,G12596,train,4e0d2c084b7642eaaf0aeab2730f3149,916aef3aa80f68c6
Normal-3690.png,Normal,Normal/images/Normal-3690.png,Normal/masks/Normal-3690.png,True,G12597,train,7ea0263e494bbbd9ce774454de6b8434,f96d961a0d1b1947
Normal-3691.png,Normal,Normal/images/Normal-3691.png,Normal/masks/Normal-3691.png,True,G12598,train,82942010cc39da29dc015da0d0985bf1,916887d33bc52ed8
Normal-3692.png,Normal,Normal/images/Normal-3692.png,Normal/masks/Normal-3692.png,True,G12599,train,febadd440f6f61860f696afbe4971a68,997d46575b0f2829
Normal-3693.png,Normal,Normal/images/Normal-3693.png,Normal/masks/Normal-3693.png,True,G12600,val,bb52b5b7b6b9f6cb0778b09e99ea4dbc,d52ac30975541df6
Normal-3694.png,Normal,Normal/images/Normal-3694.png,Normal/masks/Normal-3694.png,True,G12601,val,ba3da9e45c4c6e41a920027cea2bb2c5,dd79c25b5d681c18
Normal-3695.png,Normal,Normal/images/Normal-3695.png,Normal/masks/Normal-3695.png,True,G12602,val,6e1131cc17b0d567d48c892a96f4b06a,dd3dc21b0c4f1cca
Normal-3696.png,Normal,Normal/images/Normal-3696.png,Normal/masks/Normal-3696.png,True,G12603,test,bdcd29ddddf7997d88ef6fddc5638f2c,907f031a4d38fce9
Normal-3697.png,Normal,Normal/images/Normal-3697.png,Normal/masks/Normal-3697.png,True,G12604,train,7a4a6d310edc3760517cf484856a90be,cc3c421b452eb9ed
Normal-3698.png,Normal,Normal/images/Normal-3698.png,Normal/masks/Normal-3698.png,True,G12605,train,9d3095171447dfc2ac05ca603706d4f9,dd39c55616461e1e
Normal-3699.png,Normal,Normal/images/Normal-3699.png,Normal/masks/Normal-3699.png,True,G12606,train,c0b631e6835174ed548b1280486f3066,c17f06461a1d4d5f
Normal-37.png,Normal,Normal/images/Normal-37.png,Normal/masks/Normal-37.png,True,G12607,train,c1907d4d17c3c049d287ae4e0c5ac825,d979c61b5c4e1d42
Normal-370.png,Normal,Normal/images/Normal-370.png,Normal/masks/Normal-370.png,True,G12608,train,290f04d3c53d01077becf0ee5559c24f,907cc71ecc1b788d
Normal-3700.png,Normal,Normal/images/Normal-3700.png,Normal/masks/Normal-3700.png,True,G12609,val,03fd439611b6aec026046e4021cee00e,c82e564b1f452bb5
Normal-3701.png,Normal,Normal/images/Normal-3701.png,Normal/masks/Normal-3701.png,True,G12610,train,847a10b475807db8856238b555901e31,997b861a196d3ae1
Normal-3702.png,Normal,Normal/images/Normal-3702.png,Normal/masks/Normal-3702.png,True,G12611,train,043dfdbdeb527c5d9b8038084bd00ca5,917857171fd60e1a
Normal-3703.png,Normal,Normal/images/Normal-3703.png,Normal/masks/Normal-3703.png,True,G12612,train,da593cc88340a73a62c11e1fa8557dda,dd3cd24b11471e5c
Normal-3704.png,Normal,Normal/images/Normal-3704.png,Normal/masks/Normal-3704.png,True,G12613,train,5b0be60b3a44bc32f0a78a5244c70849,917a061e1f4b29bb
Normal-3705.png,Normal,Normal/images/Normal-3705.png,Normal/masks/Normal-3705.png,True,G12614,train,4268371897e3fee6faad0d7f80f4463a,947e43534f4f0b1a
Normal-3706.png,Normal,Normal/images/Normal-3706.png,Normal/masks/Normal-3706.png,True,G12615,train,12338b56a9b862c4a3e32696e2711e4e,ac64d20bc11f5f4d
Normal-3707.png,Normal,Normal/images/Normal-3707.png,Normal/masks/Normal-3707.png,True,G12616,train,964b3fc7f4b3f66f56db4f7cd3868e1a,803e575a1d4d3b39
Normal-3708.png,Normal,Normal/images/Normal-3708.png,Normal/masks/Normal-3708.png,True,G12617,val,57db18ca8f4efb2ca7690c7afb58710d,9939461acc6e39ec
Normal-3709.png,Normal,Normal/images/Normal-3709.png,Normal/masks/Normal-3709.png,True,G12618,train,153646b946e7f86ea8a6ef19c87ad6ad,d42bc31f4d2e384d
Normal-371.png,Normal,Normal/images/Normal-371.png,Normal/masks/Normal-371.png,True,G12619,test,8ea197eb3f14ac12ba39de8d2972d468,917fce1a194d1c5c
Normal-3710.png,Normal,Normal/images/Normal-3710.png,Normal/masks/Normal-3710.png,True,G12620,test,6b7d0b44e6e4f30ff20b1a46be5a693d,dd3dc2591b4f1d08
Normal-3711.png,Normal,Normal/images/Normal-3711.png,Normal/masks/Normal-3711.png,True,G12621,val,45ec8ad69373de28727f029312266c68,d97dc6521b4d0d1a
Normal-3712.png,Normal,Normal/images/Normal-3712.png,Normal/masks/Normal-3712.png,True,G12622,test,bd117a20a88094eb264a16a91a036f15,dc28831f4e065f79
Normal-3713.png,Normal,Normal/images/Normal-3713.png,Normal/masks/Normal-3713.png,True,G12623,train,76bd1bbb3585ae8087c3336b723024ed,c93d86561d465b1d
Normal-3714.png,Normal,Normal/images/Normal-3714.png,Normal/masks/Normal-3714.png,True,G12624,train,0e683af091cb21b3064ac8846e124149,817d061a5f797968
Normal-3715.png,Normal,Normal/images/Normal-3715.png,Normal/masks/Normal-3715.png,True,G12625,train,4d3d07eebbe94d37693b344217f42867,9979861e1d471a79
Normal-3716.png,Normal,Normal/images/Normal-3716.png,Normal/masks/Normal-3716.png,True,G12626,train,843d98e70ece05d4f0be77e3e8d6cfb3,d96cc61ad54eb885
Normal-3717.png,Normal,Normal/images/Normal-3717.png,Normal/masks/Normal-3717.png,True,G12627,train,753e47bca39e8dd59043d8eba13199c6,d979561b114b398b
Normal-3718.png,Normal,Normal/images/Normal-3718.png,Normal/masks/Normal-3718.png,True,G12628,val,1e5a99014927cbb52aebe7ce5e11e069,9d28871b4f5a594e
Normal-3719.png,Normal,Normal/images/Normal-3719.png,Normal/masks/Normal-3719.png,True,G12629,train,5ae2f4aa2dfca24b2ee921dc1d32c311,89e9861a055e7b4f
Normal-372.png,Normal,Normal/images/Normal-372.png,Normal/masks/Normal-372.png,True,G12630,train,b696bf14096b1a29856704bd77463ce4,906ae77ae168e0e5
Normal-3720.png,Normal,Normal/images/Normal-3720.png,Normal/masks/Normal-3720.png,True,G12631,train,4aa471524a1853ec5cde60bf2f1d041e,817817195f7859ec
Normal-3721.png,Normal,Normal/images/Normal-3721.png,Normal/masks/Normal-3721.png,True,G12632,train,b29bc08a45d6f81eec94904db236feba,996f461b0f4e1b45
Normal-3722.png,Normal,Normal/images/Normal-3722.png,Normal/masks/Normal-3722.png,True,G12633,train,db6dea8a9c7c138d0962889d06379664,9909165a797979e1
Normal-3723.png,Normal,Normal/images/Normal-3723.png,Normal/masks/Normal-3723.png,True,G12634,train,fe57130a83161e3d86f28c261c388814,d42e534b4b4d0f0f
Normal-3724.png,Normal,Normal/images/Normal-3724.png,Normal/masks/Normal-3724.png,True,G12635,train,379efb51141ccc47e2ac5c3f83f62f21,dd2cc20255465f3f
Normal-3725.png,Normal,Normal/images/Normal-3725.png,Normal/masks/Normal-3725.png,True,G12636,train,58fab4935cd5ed6d3a7973b7e5404f16,d93b965a1f005b39
Normal-3726.png,Normal,Normal/images/Normal-3726.png,Normal/masks/Normal-3726.png,True,G12637,train,ba7f93c0531925304e1c24601af6a6ea,95781e1b7b7ca0a4
Normal-3727.png,Normal,Normal/images/Normal-3727.png,Normal/masks/Normal-3727.png,True,G12638,train,a56b718c84806b18f9735483a3715fa0,d42dca42df52af48
Normal-3728.png,Normal,Normal/images/Normal-3728.png,Normal/masks/Normal-3728.png,True,G12639,val,7d38114e93b5db7abd2f659b249ca6b9,f97f861a164f1948
Normal-3729.png,Normal,Normal/images/Normal-3729.png,Normal/masks/Normal-3729.png,True,G12640,val,3d5e9858d70928e2031d0f2d97f3539f,d93dc45a1f4c5919
Normal-373.png,Normal,Normal/images/Normal-373.png,Normal/masks/Normal-373.png,True,G12641,train,2610e085dafa260a1b8822c0ad53f9f6,916c8678cd6d4bc6
Normal-3730.png,Normal,Normal/images/Normal-3730.png,Normal/masks/Normal-3730.png,True,G12642,test,1723d1f15ce6ea23577fd38416d6ac78,9979851e58497f38
Normal-3731.png,Normal,Normal/images/Normal-3731.png,Normal/masks/Normal-3731.png,True,G12643,test,932082a9ccb38d28e24f0a67817aca23,dd28c25b1f4caa1d
Normal-3732.png,Normal,Normal/images/Normal-3732.png,Normal/masks/Normal-3732.png,True,G12644,test,d271aa8e9e000229df540eb2a681444e,c83d421fd10f1b4f
Normal-3733.png,Normal,Normal/images/Normal-3733.png,Normal/masks/Normal-3733.png,True,G12645,train,a856af52eb4dd7f04a5656859c62620b,8178061f191f5f5e
Normal-3734.png,Normal,Normal/images/Normal-3734.png,Normal/masks/Normal-3734.png,True,G12646,train,e81707cb9e248e39d062c69354a7eef7,8178161e79e9e9a9
Normal-3735.png,Normal,Normal/images/Normal-3735.png,Normal/masks/Normal-3735.png,True,G12647,train,2a1eccf4a69849cce728cd10b56f5594,897b07561a1cf978
Normal-3736.png,Normal,Normal/images/Normal-3736.png,Normal/masks/Normal-3736.png,True,G12648,train,c2262e9f6f8850f6c76a724e4d2adf0a,dd2956171e5d093a
Normal-3737.png,Normal,Normal/images/Normal-3737.png,Normal/masks/Normal-3737.png,True,G12649,train,77a67876d25ce1578602fe0ae87af5e3,c52f76f0da4941f0
Normal-3738.png,Normal,Normal/images/Normal-3738.png,Normal/masks/Normal-3738.png,True,G12650,val,d33a5175f6e4e58fe9d648604bfd1bec,9968061a5d5f7a69
Normal-3739.png,Normal,Normal/images/Normal-3739.png,Normal/masks/Normal-3739.png,True,G12651,train,a6bbde6df7e0f9b21bd9f2b70677cb91,dd2e825f4cec2a38
Normal-374.png,Normal,Normal/images/Normal-374.png,Normal/masks/Normal-374.png,True,G12652,train,e60f988650e033ec9ae339aa2874c722,906ac63ac94d7ccd
Normal-3740.png,Normal,Normal/images/Normal-3740.png,Normal/masks/Normal-3740.png,True,G12653,train,0ffbd33cf1c946274cf95ea420e7ef58,c06b561a0f2b3d3d
Normal-3741.png,Normal,Normal/images/Normal-3741.png,Normal/masks/Normal-3741.png,True,G12654,val,d3cd075cc87b1c7585d9c94a4de7565a,d92f561d4c29394d
Normal-3742.png,Normal,Normal/images/Normal-3742.png,Normal/masks/Normal-3742.png,True,G12655,train,4ccedd1b0bab8869455f574680267a79,993d84c31b1f471b
Normal-3743.png,Normal,Normal/images/Normal-3743.png,Normal/masks/Normal-3743.png,True,G12656,train,a6d2fa47d2c54f140703ebc998290d5f,992e464e5e447e69
Normal-3744.png,Normal,Normal/images/Normal-3744.png,Normal/masks/Normal-3744.png,True,G12657,train,76707f79d0988cbb4445e762e59aa7c6,d939869e1d0e5f18
Normal-3745.png,Normal,Normal/images/Normal-3745.png,Normal/masks/Normal-3745.png,True,G12658,train,149ed6f80db8ff40f8778f29d4db85a3,917806161e5b5f5b
Normal-3746.png,Normal,Normal/images/Normal-3746.png,Normal/masks/Normal-3746.png,True,G12659,val,51036feb6f2aca24e55e34d7f95ea120,8978971e1a895b79
Normal-3747.png,Normal,Normal/images/Normal-3747.png,Normal/masks/Normal-3747.png,True,G12660,test,99a0fb7e90ab059648474735879ac4bc,887f87161e555e1c
Normal-3748.png,Normal,Normal/images/Normal-3748.png,Normal/masks/Normal-3748.png,True,G12661,train,151cdd3f4b400d3e74096190034794bf,9969c65e5ec64f08
Normal-3749.png,Normal,Normal/images/Normal-3749.png,Normal/masks/Normal-3749.png,True,G12662,test,16d9ac104026204f694602a762d818e0,997d861e1d4f1a19
Normal-375.png,Normal,Normal/images/Normal-375.png,Normal/masks/Normal-375.png,True,G12663,train,90c12155ae9fce0bd903db49b23d32fc,d4ee871b494558d6
Normal-3750.png,Normal,Normal/images/Normal-3750.png,Normal/masks/Normal-3750.png,True,G12664,val,c0fc4a4a46178196c329ee7c74e90579,bc5cc35b4d0f190d
Normal-3751.png,Normal,Normal/images/Normal-3751.png,Normal/masks/Normal-3751.png,True,G12665,test,0a07cba769467e1bdd3c3cf6cafe6e50,8879d61e055a1f5e
Normal-3752.png,Normal,Normal/images/Normal-3752.png,Normal/masks/Normal-3752.png,True,G12666,val,2fde2bc6d7f0ed0aa8c65a49a72b8ec2,992d8612555f5ad9
Normal-3753.png,Normal,Normal/images/Normal-3753.png,Normal/masks/Normal-3753.png,True,G12667,train,7cef8d5634612e8c24293bf217ca94bf,943c474b0b4b2f7c
Normal-3754.png,Normal,Normal/images/Normal-3754.png,Normal/masks/Normal-3754.png,True,G12668,test,87b2bfcf39502fe9869d7d9ed5c8d8db,c2348f4d6c6eaa6c
Normal-3755.png,Normal,Normal/images/Normal-3755.png,Normal/masks/Normal-3755.png,True,G12669,train,8b1044a55a88aaedb10d8ea61b4b1e9d,dc2f421f0d443fe8
Normal-3756.png,Normal,Normal/images/Normal-3756.png,Normal/masks/Normal-3756.png,True,G12670,train,92242db28d20a527c48256daed946e41,997947061a4d5f5a
Normal-3757.png,Normal,Normal/images/Normal-3757.png,Normal/masks/Normal-3757.png,True,G12671,train,aa552405f84d9cfa7618bc768dde8363,b979431f474f1849
Normal-3758.png,Normal,Normal/images/Normal-3758.png,Normal/masks/Normal-3758.png,True,G12672,train,0ab55db62d3afdbb3e53d257828c45bb,d139021f195b6d79
Normal-3759.png,Normal,Normal/images/Normal-3759.png,Normal/masks/Normal-3759.png,True,G12673,train,29206913b9a7d82a3429961dfac3b9f0,d57906075a1aec79
Normal-376.png,Normal,Normal/images/Normal-376.png,Normal/masks/Normal-376.png,True,G12674,train,137e1c011658541d19bb1208b290687a,94bec31bd54614d6
Normal-3760.png,Normal,Normal/images/Normal-3760.png,Normal/masks/Normal-3760.png,True,G12675,train,3bfcef9ccb3f90e2fe5c1fc17d1f51df,c83f464b1d4d1aec
Normal-3761.png,Normal,Normal/images/Normal-3761.png,Normal/masks/Normal-3761.png,True,G12676,train,d31934197ea8164580a91e7ece4901e9,996c461b596f78a1
Normal-3762.png,Normal,Normal/images/Normal-3762.png,Normal/masks/Normal-3762.png,True,G12677,train,347a5a74723e5f11398f126425343ee5,9779d07a702e642e
Normal-3763.png,Normal,Normal/images/Normal-3763.png,Normal/masks/Normal-3763.png,True,G12678,train,6f3502039e472fd2e700edab62bd9231,dc0e470c6c6cb4d7
Normal-3764.png,Normal,Normal/images/Normal-3764.png,Normal/masks/Normal-3764.png,True,G12679,train,8873c0d85de872b425d4fb00ddfd0899,d908521f1f2d38be
Normal-3765.png,Normal,Normal/images/Normal-3765.png,Normal/masks/Normal-3765.png,True,G12680,val,57b09da7b00ceac723a49e3b16ee5524,9979061e1f867ab8
Normal-3766.png,Normal,Normal/images/Normal-3766.png,Normal/masks/Normal-3766.png,True,G12681,test,63bba90153ff210837ffb88b1fee87ce,d43c43cb2cca2fcc
Normal-3767.png,Normal,Normal/images/Normal-3767.png,Normal/masks/Normal-3767.png,True,G12682,test,03661e2d1db06caca1fcc5ea61906e97,dd2c464e18edee90
Normal-3768.png,Normal,Normal/images/Normal-3768.png,Normal/masks/Normal-3768.png,True,G12683,test,6d593e2d75f9036fd31a950e137f6848,8178571e165e5a79
Normal-3769.png,Normal,Normal/images/Normal-3769.png,Normal/masks/Normal-3769.png,True,G12684,train,6908bc3f79fbff3427361e4fd5294c8a,dd2cd253405f5c4b
Normal-377.png,Normal,Normal/images/Normal-377.png,Normal/masks/Normal-377.png,True,G12685,train,07c282d1985746b7ced303485c19f4b7,906be61a59591ddc
Normal-3770.png,Normal,Normal/images/Normal-3770.png,Normal/masks/Normal-3770.png,True,G12686,train,a2bc9f38406c80582cff816052a4ebc7,dd3cc25b0b4d0b1b
Normal-3771.png,Normal,Normal/images/Normal-3771.png,Normal/masks/Normal-3771.png,True,G12687,train,0f7cf92b33db9463ebf50504571ac1fe,dc2ec21f0f4b1f48
Normal-3772.png,Normal,Normal/images/Normal-3772.png,Normal/masks/Normal-3772.png,True,G12688,train,f4fba9ec69bbf36631560c055186f9b7,dd68d24b4d4a1a6e
Normal-3773.png,Normal,Normal/images/Normal-3773.png,Normal/masks/Normal-3773.png,True,G12689,train,e86bf92a614f1d2928606e86496c55c3,dd2ed30b444f1c4e
Normal-3774.png,Normal,Normal/images/Normal-3774.png,Normal/masks/Normal-3774.png,True,G12690,train,4e975b4cd1e7d8e4e57f68913b9f9f8b,dc2ed30b0d4a2e6d
Normal-3775.png,Normal,Normal/images/Normal-3775.png,Normal/masks/Normal-3775.png,True,G12691,train,5a60aa165416716719f9282a750ca57b,cc3d421b0f4f1e4e
Normal-3776.png,Normal,Normal/images/Normal-3776.png,Normal/masks/Normal-3776.png,True,G12692,train,5a6a0b9ac48c007861a683c0e426ee9b,9968d6581d4d3997
Normal-3777.png,Normal,Normal/images/Normal-3777.png,Normal/masks/Normal-3777.png,True,G12693,train,d6c02c632b017439bbdbe5a6decce8b7,dc5c460b1d4cae3d
Normal-3778.png,Normal,Normal/images/Normal-3778.png,Normal/masks/Normal-3778.png,True,G12694,train,cc05a5222b4c8c4c57cfb59bf94d93dd,887d851e1a4f7e58
Normal-3779.png,Normal,Normal/images/Normal-3779.png,Normal/masks/Normal-3779.png,True,G12695,val,a615e192abebec5b9afe7f7c4f8b9f1f,9979061e1b0d7b39
Normal-378.png,Normal,Normal/images/Normal-378.png,Normal/masks/Normal-378.png,True,G12696,train,df6c31931f75c955fc0301759ede9e70,d47e871e8d0d6c86
Normal-3780.png,Normal,Normal/images/Normal-3780.png,Normal/masks/Normal-3780.png,True,G12697,val,b5e7d97dba6cb4912ab7908bb288ef68,dd2e530b5d0c4e5c
Normal-3781.png,Normal,Normal/images/Normal-3781.png,Normal/masks/Normal-3781.png,True,G12698,train,a147ece3e39df036319453a2904ae0e9,937a56151a0bea3b
Normal-3782.png,Normal,Normal/images/Normal-3782.png,Normal/masks/Normal-3782.png,True,G12699,test,980c3aa8b3fdc242c693deac97ad3616,dd3d4b43194b0f38
Normal-3783.png,Normal,Normal/images/Normal-3783.png,Normal/masks/Normal-3783.png,True,G12700,test,2ca857c989d1deb344baa47ed3f34da0,dc2ec3530f4e4d0d
Normal-3784.png,Normal,Normal/images/Normal-3784.png,Normal/masks/Normal-3784.png,True,G12701,train,79ea168f598f7ffc623deacc258467de,dc2d474b1d0d1d1d
Normal-3785.png,Normal,Normal/images/Normal-3785.png,Normal/masks/Normal-3785.png,True,G12702,train,375e092699cf603dcccb10f46ce2fd1d,836b5017945e1d5f
Normal-3786.png,Normal,Normal/images/Normal-3786.png,Normal/masks/Normal-3786.png,True,G12703,val,58839faf855a03227aff7f93919d49f3,893b464a1cece979
Normal-3787.png,Normal,Normal/images/Normal-3787.png,Normal/masks/Normal-3787.png,True,G12704,train,95f3b31684111b57ce7fc335c5615716,9959861e18e9a5be
Normal-3788.png,Normal,Normal/images/Normal-3788.png,Normal/masks/Normal-3788.png,True,G12705,val,05aadf0576160257c5466053d6c7f42a,996a161a875a5b5e
Normal-3789.png,Normal,Normal/images/Normal-3789.png,Normal/masks/Normal-3789.png,True,G12706,test,057bd5840a4451f07bed194adaa2d245,d87b45861a0f5e1e
Normal-379.png,Normal,Normal/images/Normal-379.png,Normal/masks/Normal-379.png,True,G12707,train,37e2757b71dccf1c4ca8ee1982bc4876,9128c63af86d396d
Normal-3790.png,Normal,Normal/images/Normal-3790.png,Normal/masks/Normal-3790.png,True,G12708,train,0a1338191a7c0940380a1b38adf3e915,d96b421e0d2d38f9
Normal-3791.png,Normal,Normal/images/Normal-3791.png,Normal/masks/Normal-3791.png,True,G12709,train,84355c3e3bf54de9feceb92ea62ae2cf,993f061a594c78f9
Normal-3792.png,Normal,Normal/images/Normal-3792.png,Normal/masks/Normal-3792.png,True,G12710,train,ff0b7a07ac1497539e98e2c80f37180d,99e98618960e3f6e
Normal-3793.png,Normal,Normal/images/Normal-3793.png,Normal/masks/Normal-3793.png,True,G12711,train,459cc476d6c45803e8e969230cdd6525,d03d4607191f4f4f
Normal-3794.png,Normal,Normal/images/Normal-3794.png,Normal/masks/Normal-3794.png,True,G12712,train,6b78e5ff6f61c8954ed521e629911a70,853e870f0d953d2d
Normal-3795.png,Normal,Normal/images/Normal-3795.png,Normal/masks/Normal-3795.png,True,G12713,train,f0876b68cb8a74f9eec514db05c9bf1f,f97946561a1d0d0f
Normal-3796.png,Normal,Normal/images/Normal-3796.png,Normal/masks/Normal-3796.png,True,G12714,train,15b7c5f6475510a3bffcbb893de8c2be,917806165b5f785b
Normal-3797.png,Normal,Normal/images/Normal-3797.png,Normal/masks/Normal-3797.png,True,G12715,train,9024e2342fa648161f57bd7e9f7f694d,9c75431f430f383e
Normal-3798.png,Normal,Normal/images/Normal-3798.png,Normal/masks/Normal-3798.png,True,G12716,train,e87679e4177e3bde49a049d5675a55ba,d939c21f400fbacb
Normal-3799.png,Normal,Normal/images/Normal-3799.png,Normal/masks/Normal-3799.png,True,G12717,train,98ae5f21b2b222ba08d505fbd05244d0,dd2ec31b5d4a1c49
Normal-38.png,Normal,Normal/images/Normal-38.png,Normal/masks/Normal-38.png,True,G12718,train,cc4cb31a0d42b2c756763b28dbe4bf07,dd4d531f4d0c1e0b
Normal-380.png,Normal,Normal/images/Normal-380.png,Normal/masks/Normal-380.png,True,G12719,val,1249f7918488149121135245013cf714,916f875a894dc8c7
Normal-3800.png,Normal,Normal/images/Normal-3800.png,Normal/masks/Normal-3800.png,True,G12720,val,ed5f606eacd140106178ce99efc786fa,fc2dc21b074b1f4c
Normal-3801.png,Normal,Normal/images/Normal-3801.png,Normal/masks/Normal-3801.png,True,G12721,test,1487bba6fd9438419295e447d8176a1b,983dc5161e455f1d
Normal-3802.png,Normal,Normal/images/Normal-3802.png,Normal/masks/Normal-3802.png,True,G12722,test,646e03aa72a71053f02d8df6af645fc5,81785e171b5ee89a
Normal-3803.png,Normal,Normal/images/Normal-3803.png,Normal/masks/Normal-3803.png,True,G12723,train,2305885e001afbe424bd7e4c3051755b,d4afc34bc20f0b0f
Normal-3804.png,Normal,Normal/images/Normal-3804.png,Normal/masks/Normal-3804.png,True,G12724,train,dceabaecd6b214952e0e7535600904b1,9360851d8f5d3c5e
Normal-3805.png,Normal,Normal/images/Normal-3805.png,Normal/masks/Normal-3805.png,True,G12725,train,8fa3dbb4b680c03abf317a4b83c0b09a,d90d861e596a38eb
Normal-3806.png,Normal,Normal/images/Normal-3806.png,Normal/masks/Normal-3806.png,True,G12726,train,67a4aea40defb1d4f3923881a4bf258e,dd28d25315c74b59
Normal-3807.png,Normal,Normal/images/Normal-3807.png,Normal/masks/Normal-3807.png,True,G12727,train,cc6ca418d09c16a530c7efcefbbeb56d,d979031b19291ebd
Normal-3808.png,Normal,Normal/images/Normal-3808.png,Normal/masks/Normal-3808.png,True,G12728,train,42fdf2f862b7a2fcacd1add1a0a5cf7a,c87d475f081f091f
Normal-3809.png,Normal,Normal/images/Normal-3809.png,Normal/masks/Normal-3809.png,True,G12729,train,63d7691c6c4cc47c6d923df6490866d3,c83dc25f164e1f13
Normal-381.png,Normal,Normal/images/Normal-381.png,Normal/masks/Normal-381.png,True,G12730,train,b08789929789ac3734311a67853dc106,d4ee870bcda468c3
Normal-3810.png,Normal,Normal/images/Normal-3810.png,Normal/masks/Normal-3810.png,True,G12731,train,d2ad2864274502577a56f75ad706f035,9d7cc0139b470f1d
Normal-3811.png,Normal,Normal/images/Normal-3811.png,Normal/masks/Normal-3811.png,True,G12732,train,0ce9998b5a3f25a025c52241b1b86cb4,993e16185b69e8e9
Normal-3812.png,Normal,Normal/images/Normal-3812.png,Normal/masks/Normal-3812.png,True,G12733,val,032ee3bb2770d70eaaf13a796f925ce6,c1295e9e182de597
Normal-3813.png,Normal,Normal/images/Normal-3813.png,Normal/masks/Normal-3813.png,True,G12734,train,5d324a0b234da3c5c7704714a063357d,817817871e1afaea
Normal-3814.png,Normal,Normal/images/Normal-3814.png,Normal/masks/Normal-3814.png,True,G12735,train,bd298f4b4c1857f9dfc4c1e3be2229cf,dd2f471e441e184f
Normal-3815.png,Normal,Normal/images/Normal-3815.png,Normal/masks/Normal-3815.png,True,G12736,train,72d5014db332468129b0b4b682ebc55f,997987121e5e385b
Normal-3816.png,Normal,Normal/images/Normal-3816.png,Normal/masks/Normal-3816.png,True,G12737,train,fad9cc92a7c121e1514e99a81c9ef4e3,ae64871b83dab9c2
Normal-3817.png,Normal,Normal/images/Normal-3817.png,Normal/masks/Normal-3817.png,True,G12738,val,e21c16d7ba740931e5d84cee383c44df,993f061b5959594b
Normal-3818.png,Normal,Normal/images/Normal-3818.png,Normal/masks/Normal-3818.png,True,G12739,val,0704b6372ad41034fe9821055c0436f9,b961c71e4a1f1e49
Normal-3819.png,Normal,Normal/images/Normal-3819.png,Normal/masks/Normal-3819.png,True,G12740,test,ae477ebdfdc2c57cfbfe5d69c422a9e5,997b46161e4d0b3b
Normal-382.png,Normal,Normal/images/Normal-382.png,Normal/masks/Normal-382.png,True,G12741,test,c7c6a15e18acfa4edaaab2be8545bba5,d07faf1a890d2987
Normal-3820.png,Normal,Normal/images/Normal-3820.png,Normal/masks/Normal-3820.png,True,G12742,train,6c4fb10f3983b44b7729c06a0716a29f,81781e161a5f7a3b
Normal-3821.png,Normal,Normal/images/Normal-3821.png,Normal/masks/Normal-3821.png,True,G12743,test,1eca08bd5f0d35cce6065b8c54c2fc56,c83dc30b4e6a7b38
Normal-3822.png,Normal,Normal/images/Normal-3822.png,Normal/masks/Normal-3822.png,True,G12744,train,b7fcea6a3afba6d1dfd6ad58a23cbcd8,885f061e194d7b79
Normal-3823.png,Normal,Normal/images/Normal-3823.png,Normal/masks/Normal-3823.png,True,G12745,train,03fef5a2274cb3d0b2a130a65596e08b,9978065a1b6df8a3
Normal-3824.png,Normal,Normal/images/Normal-3824.png,Normal/masks/Normal-3824.png,True,G12746,train,e9dbd9af6ea58eb53189bed5fcd0fa8c,dc3852435b4b4b1f
Normal-3825.png,Normal,Normal/images/Normal-3825.png,Normal/masks/Normal-3825.png,True,G12747,train,d1fa24f8dc0db968b4a5dfdcb3b9c719,d13dc61a58ed38e8
Normal-3826.png,Normal,Normal/images/Normal-3826.png,Normal/masks/Normal-3826.png,True,G12748,train,5f816b89f918cc60f0645eceff8335ee,902b579f5a4d3998
Normal-3827.png,Normal,Normal/images/Normal-3827.png,Normal/masks/Normal-3827.png,True,G12749,train,dda1814113e10764f389471317c7014f,992b061e165e5f5c
Normal-3828.png,Normal,Normal/images/Normal-3828.png,Normal/masks/Normal-3828.png,True,G12750,train,6ffac54d9d6e8ec1783b53d7be2853ad,9968871a4d5f59c5
Normal-3829.png,Normal,Normal/images/Normal-3829.png,Normal/masks/Normal-3829.png,True,G12751,train,ae53fa28ceca368523d90cf13ded4cfb,f929c65a18475a5d
Normal-383.png,Normal,Normal/images/Normal-383.png,Normal/masks/Normal-383.png,True,G12752,train,43cb8d49b792d6b67da5c1ab48be94b2,d06ec74badc56c24
Normal-3830.png,Normal,Normal/images/Normal-3830.png,Normal/masks/Normal-3830.png,True,G12753,train,f58ef2af73b8cf7fa5da416aedfe311e,9c2ed34b4d4c1d4e
Normal-3831.png,Normal,Normal/images/Normal-3831.png,Normal/masks/Normal-3831.png,True,G12754,train,0fc7516cd4216463aef5585a0bef1a96,dd2ad31f401b594b
Normal-3832.png,Normal,Normal/images/Normal-3832.png,Normal/masks/Normal-3832.png,True,G12755,train,22a4beae7d41e673cfaead1db2ddea5e,c47f4b0f0f0e0e2d
Normal-3833.png,Normal,Normal/images/Normal-3833.png,Normal/masks/Normal-3833.png,True,G12756,train,975186c4ec8611787b9bb9df7c07f368,c87907861a1b4f5f
Normal-3834.png,Normal,Normal/images/Normal-3834.png,Normal/masks/Normal-3834.png,True,G12757,val,168687ff4a94ab1b5f7ffddcfdb1791a,8179d656114b2f3d
Normal-3835.png,Normal,Normal/images/Normal-3835.png,Normal/masks/Normal-3835.png,True,G12758,val,44e0133c8a43c82c6f32204b9d4fa3d4,887d871a4d1a79c7
Normal-3836.png,Normal,Normal/images/Normal-3836.png,Normal/masks/Normal-3836.png,True,G12759,test,a5f07534e960bf1ea5af1aab6ae45d6f,d92d061b4d4e59e5
Normal-3837.png,Normal,Normal/images/Normal-3837.png,Normal/masks/Normal-3837.png,True,G12760,val,9e232403a43dc654a02f7fd55dea3569,996fc61a4c5a384f
Normal-3838.png,Normal,Normal/images/Normal-3838.png,Normal/masks/Normal-3838.png,True,G12761,test,63c9d137c73cfc505263d63c4afa5ed0,fc2c430f4a2d4a6f
Normal-3839.png,Normal,Normal/images/Normal-3839.png,Normal/masks/Normal-3839.png,True,G12762,test,668d38b3b9c22f306e9496e10fe0ab23,c939c61a1a0f3dbc
Normal-384.png,Normal,Normal/images/Normal-384.png,Normal/masks/Normal-384.png,True,G12763,train,3b1977868d6f2e52f89a74fa885fb5f7,956c8f3b8d2c6c4c
Normal-3840.png,Normal,Normal/images/Normal-3840.png,Normal/masks/Normal-3840.png,True,G12764,train,a5bed80aeb0fddca7ace3344124a9baf,cc0c12c35f7df0b8
Normal-3841.png,Normal,Normal/images/Normal-3841.png,Normal/masks/Normal-3841.png,True,G12765,train,86d7fd1fd3b62bba4936709dc3478dd8,dc3d83435f4ac30b
Normal-3842.png,Normal,Normal/images/Normal-3842.png,Normal/masks/Normal-3842.png,True,G12766,train,cd702a9e571cf8c8b590eb00aee2301a,997817125be171f1
Normal-3843.png,Normal,Normal/images/Normal-3843.png,Normal/masks/Normal-3843.png,True,G12767,train,acd1f40c66da8fd516e8e1f4d4ee7c3a,9d2c061b53477e1b
Normal-3844.png,Normal,Normal/images/Normal-3844.png,Normal/masks/Normal-3844.png,True,G12768,train,d64717b289cd6b5e761e02ebd08de13d,dd7a535b1f4c081a
Normal-3845.png,Normal,Normal/images/Normal-3845.png,Normal/masks/Normal-3845.png,True,G12769,train,9429020b4b1c785f1b03c9b7cd22c80f,91691e111b4ff9b8
Normal-3846.png,Normal,Normal/images/Normal-3846.png,Normal/masks/Normal-3846.png,True,G12770,train,dad7f43043965e01c0fb78dcbcd6ae8c,897f061e175e1959
Normal-3847.png,Normal,Normal/images/Normal-3847.png,Normal/masks/Normal-3847.png,True,G12771,train,0bdd167dcb2f3cd85232a3a0ea86688e,e96dc616465a554e
Normal-3848.png,Normal,Normal/images/Normal-3848.png,Normal/masks/Normal-3848.png,True,G12772,train,804d4f6f9dd019a6fc6c4cdacfe437f4,c875421b5a4b4f2f
Normal-3849.png,Normal,Normal/images/Normal-3849.png,Normal/masks/Normal-3849.png,True,G12773,train,eaa521ac6dd864c155d84ac5019a6c54,91e9ce56594d1a39
Normal-385.png,Normal,Normal/images/Normal-385.png,Normal/masks/Normal-385.png,True,G12774,val,d80e115e3a842338d76f8216efeb1004,dd2ae20fde681dc0
Normal-3850.png,Normal,Normal/images/Normal-3850.png,Normal/masks/Normal-3850.png,True,G12775,train,a0cf6e933abb55ec6e308f2e92d4c5c5,dd2cc34b4d4a4e2e
Normal-3851.png,Normal,Normal/images/Normal-3851.png,Normal/masks/Normal-3851.png,True,G12776,val,f42687241c3db5307cdf27c56a55068a,993f064e4c4c5dda
Normal-3852.png,Normal,Normal/images/Normal-3852.png,Normal/masks/Normal-3852.png,True,G12777,test,f58cddee541cc2cf32bdc52753100274,dc3d430d0e0a1dbf
Normal-3853.png,Normal,Normal/images/Normal-3853.png,Normal/masks/Normal-3853.png,True,G12778,train,b39508ba18860acf53575fd778e1c7ea,991d471b1c4b3bac
Normal-3854.png,Normal,Normal/images/Normal-3854.png,Normal/masks/Normal-3854.png,True,G12779,train,76f7de9895ef69227bcbaf16b51b058b,8879e7c61a815f9a
Normal-3855.png,Normal,Normal/images/Normal-3855.png,Normal/masks/Normal-3855.png,True,G12780,test,7b403e35b02c82df1e304e25b3d3e154,c43943194a2fbb6e
Normal-3856.png,Normal,Normal/images/Normal-3856.png,Normal/masks/Normal-3856.png,True,G12781,test,ca1c4e6998691334849422ef6a3f14ad,9d7e40c31b514f0f
Normal-3857.png,Normal,Normal/images/Normal-3857.png,Normal/masks/Normal-3857.png,True,G12782,train,e2f4d1db5beaec73e37566fcd8e12dc8,c979565b54ca1bc8
Normal-3858.png,Normal,Normal/images/Normal-3858.png,Normal/masks/Normal-3858.png,True,G12783,val,e9625df0c6b7422804e320c05f84a69c,d86901869e1b9fc7
Normal-3859.png,Normal,Normal/images/Normal-3859.png,Normal/masks/Normal-3859.png,True,G12784,train,4c9b254308cc99e2fb7ddc6c6f32d2a6,94c96b9e61719663
Normal-386.png,Normal,Normal/images/Normal-386.png,Normal/masks/Normal-386.png,True,G12785,train,ee2190562e88c386afcc36e3de1709ea,d02dc639e9293c6d
Normal-3860.png,Normal,Normal/images/Normal-3860.png,Normal/masks/Normal-3860.png,True,G12786,train,82e2bc72c8f442cd1317c8a3517116b6,8179169a5b796879
Normal-3861.png,Normal,Normal/images/Normal-3861.png,Normal/masks/Normal-3861.png,True,G12787,train,9b40dc5fa2e9d1d888a36ae5100ce75f,fd45c20f4d1e2d2c
Normal-3862.png,Normal,Normal/images/Normal-3862.png,Normal/masks/Normal-3862.png,True,G12788,train,f5273943554040acbd52684c4789653e,dc2c42530a5e5f79
Normal-3863.png,Normal,Normal/images/Normal-3863.png,Normal/masks/Normal-3863.png,True,G12789,train,c33a8be9207554e043a0793a6d17d62b,897f061a1a4d7b59
Normal-3864.png,Normal,Normal/images/Normal-3864.png,Normal/masks/Normal-3864.png,True,G12790,train,3ab3c8b18f9780ba351bb9f22704f9d5,f927061b4619c55f
Normal-3865.png,Normal,Normal/images/Normal-3865.png,Normal/masks/Normal-3865.png,True,G12791,train,c6ca9069c761283822fb4c7a2032ffc4,817907161a5d797b
Normal-3866.png,Normal,Normal/images/Normal-3866.png,Normal/masks/Normal-3866.png,True,G12792,train,72c3642fe5cc230c62bec738eeeb1a3a,cc3d43480f6d3db8
Normal-3867.png,Normal,Normal/images/Normal-3867.png,Normal/masks/Normal-3867.png,True,G12793,train,927b59104667b9107ad0ca71e121aa9e,807857865b5d2d3e
Normal-3868.png,Normal,Normal/images/Normal-3868.png,Normal/masks/Normal-3868.png,True,G12794,train,50c9d5bd1b4d15d96dd9e5aa126eccc0,8969c71e10ce7ad5
Normal-3869.png,Normal,Normal/images/Normal-3869.png,Normal/masks/Normal-3869.png,True,G12795,val,2b2376e497fa88d2eaaf013b9e022e7a,d97d86121fc55a19
Normal-387.png,Normal,Normal/images/Normal-387.png,Normal/masks/Normal-387.png,True,G12796,train,bd1a8f7488f227e510a4a858488d7fe3,d93ad65f1c411d86
Normal-3870.png,Normal,Normal/images/Normal-3870.png,Normal/masks/Normal-3870.png,True,G12797,train,f4899f935203f6a3dfd58ff8c09810ed,897d87921e84f999
Normal-3871.png,Normal,Normal/images/Normal-3871.png,Normal/masks/Normal-3871.png,True,G12798,val,5001589a13868dae4d5c6af98645689a,9968d61e1e8e38b9
Normal-3872.png,Normal,Normal/images/Normal-3872.png,Normal/masks/Normal-3872.png,True,G12799,val,4a68e63a67ac3e65325a601e9da8b45d,cd0cd3cb0f2dd02d
Normal-3873.png,Normal,Normal/images/Normal-3873.png,Normal/masks/Normal-3873.png,True,G12800,test,0d8c0566be9494a3a737cc8ea159f939,c83d421f1b5b0f3a
Normal-3874.png,Normal,Normal/images/Normal-3874.png,Normal/masks/Normal-3874.png,True,G12801,test,9f0a0590df6d7f48affdcac814a96365,917d06471b0b4d3f
Normal-3875.png,Normal,Normal/images/Normal-3875.png,Normal/masks/Normal-3875.png,True,G12802,test,64d30113cbdbcedcedccceaa97b8a38b,dd3cd20b5e4b1c4c
Normal-3876.png,Normal,Normal/images/Normal-3876.png,Normal/masks/Normal-3876.png,True,G12803,test,01c51d70a9fb3a85f0127a357c091fc3,897f070a1f4d7a31
Normal-3877.png,Normal,Normal/images/Normal-3877.png,Normal/masks/Normal-3877.png,True,G12804,train,0b9b38845050418c84269c8f4bfd7e71,dd3f520f4d490b19
Normal-3878.png,Normal,Normal/images/Normal-3878.png,Normal/masks/Normal-3878.png,True,G12805,train,af2c38d1d6d7f0d42c0267c1e1c4f5e6,896cc6861a43f71f
Normal-3879.png,Normal,Normal/images/Normal-3879.png,Normal/masks/Normal-3879.png,True,G12806,train,77661b8f386c7f02cf8af698df503d9e,992cc61a0d4f3aed
Normal-388.png,Normal,Normal/images/Normal-388.png,Normal/masks/Normal-388.png,True,G12807,train,bc4203ad471f5d1d3c25f4648eb741bb,811bc69ab8c933ed
Normal-3880.png,Normal,Normal/images/Normal-3880.png,Normal/masks/Normal-3880.png,True,G12808,train,2bf958d2b5b6e511d2510ab079b81798,897f07161b49595d
Normal-3881.png,Normal,Normal/images/Normal-3881.png,Normal/masks/Normal-3881.png,True,G12809,train,d64c8ce63600c0bcaff3fe35ff5c23d8,d57d06121b4f0e3b
Normal-3882.png,Normal,Normal/images/Normal-3882.png,Normal/masks/Normal-3882.png,True,G12810,train,92d15bf3a986eea6c39e05262ead5683,d02d42c71f5aed0b
Normal-3883.png,Normal,Normal/images/Normal-3883.png,Normal/masks/Normal-3883.png,True,G12811,val,e97427b4c7d307fa5b6930fcb24be17f,9968165a595d7969
Normal-3884.png,Normal,Normal/images/Normal-3884.png,Normal/masks/Normal-3884.png,True,G12812,train,e3af32b3005947a3f4b91a5da2f03d7d,9979471692571f1a
Normal-3885.png,Normal,Normal/images/Normal-3885.png,Normal/masks/Normal-3885.png,True,G12813,train,0c08e7231936c28d0b57a3f190233207,c46e87490dee1cec
Normal-3886.png,Normal,Normal/images/Normal-3886.png,Normal/masks/Normal-3886.png,True,G12814,test,736431ba62d5409baa9d8c3c3e5f76d6,c17b56061e165e3b
Normal-3887.png,Normal,Normal/images/Normal-3887.png,Normal/masks/Normal-3887.png,True,G12815,train,bdaf4fc84c953a6ecbe25228fd44c5f9,b97d87121b851f58
Normal-3888.png,Normal,Normal/images/Normal-3888.png,Normal/masks/Normal-3888.png,True,G12816,train,e7671c56858e699f4a31b2b4d184e9be,91fb8610974f1747
Normal-3889.png,Normal,Normal/images/Normal-3889.png,Normal/masks/Normal-3889.png,True,G12817,train,187dbf0f5a87d42badaf6cca28c4afb6,817a841c551b5f5f
Normal-389.png,Normal,Normal/images/Normal-389.png,Normal/masks/Normal-389.png,True,G12818,train,ab3df53336567516db1cf34e0668fac7,817cc65a9cce3995
Normal-3890.png,Normal,Normal/images/Normal-3890.png,Normal/masks/Normal-3890.png,True,G12819,train,2f91d8976ff9f2e07e01ebd07b59202b,dc3fd0d34bc5404b
Normal-3891.png,Normal,Normal/images/Normal-3891.png,Normal/masks/Normal-3891.png,True,G12820,val,f92d8e58f18abd695d39be813c2a1bfb,993956591b6a39e1
Normal-3892.png,Normal,Normal/images/Normal-3892.png,Normal/masks/Normal-3892.png,True,G12821,val,d3308dbe02d60c296a7cf6ec12b7e6d4,cd2cd4521b4f0b1f
Normal-3893.png,Normal,Normal/images/Normal-3893.png,Normal/masks/Normal-3893.png,True,G12822,test,d84e94e89ac1fa601c6b279c52eb44f9,d42f425f1a0f1f2c
Normal-3894.png,Normal,Normal/images/Normal-3894.png,Normal/masks/Normal-3894.png,True,G12823,train,01ceb10e03d42983625b2203df802de3,997b87961e055e16
Normal-3895.png,Normal,Normal/images/Normal-3895.png,Normal/masks/Normal-3895.png,True,G12824,train,7e53383966a7c7109160ce5268b89638,cc3d42531f430f1f
Normal-3896.png,Normal,Normal/images/Normal-3896.png,Normal/masks/Normal-3896.png,True,G12825,train,06092be212ac7265c34559c109824d0d,9968161b9d7a39e1
Normal-3897.png,Normal,Normal/images/Normal-3897.png,Normal/masks/Normal-3897.png,True,G12826,train,c8182ba7dde2a3ff51e0ace984f441ba,993b861a1b6d3a78
Normal-3898.png,Normal,Normal/images/Normal-3898.png,Normal/masks/Normal-3898.png,True,G12827,train,0ef2c29db1fdb45454a306bc4570f77a,993f861e1c0f1b1d
Normal-3899.png,Normal,Normal/images/Normal-3899.png,Normal/masks/Normal-3899.png,True,G12828,train,b8fc07b4b241c6bfffc9167a87d5a719,817b161e1e1f1b3a
Normal-39.png,Normal,Normal/images/Normal-39.png,Normal/masks/Normal-39.png,True,G12829,train,d3ace391749887dfc643336aa0a8a7d8,9979061a0d2fbda5
Normal-390.png,Normal,Normal/images/Normal-390.png,Normal/masks/Normal-390.png,True,G12830,train,a26f488e328bc05be872dba57c88b3b2,d13dc71a9c4939d4
Normal-3900.png,Normal,Normal/images/Normal-3900.png,Normal/masks/Normal-3900.png,True,G12831,train,f5bc8958c635397dca3924046c43253c,81798706124f7f5d
Normal-3901.png,Normal,Normal/images/Normal-3901.png,Normal/masks/Normal-3901.png,True,G12832,train,548a0b7132120004ee9d386479b8506e,9868c31fd71f184d
Normal-3902.png,Normal,Normal/images/Normal-3902.png,Normal/masks/Normal-3902.png,True,G12833,train,ec387a9e56bebcaa77d2eb21873324ab,9939c6521d565f38
Normal-3903.png,Normal,Normal/images/Normal-3903.png,Normal/masks/Normal-3903.png,True,G12834,train,afeffb3411d9f5e4aafa1a96eb0245c8,ad0cc24b4cec78fc
Normal-3904.png,Normal,Normal/images/Normal-3904.png,Normal/masks/Normal-3904.png,True,G12835,val,c1ef47caa8cefd9881143e239ac985d6,903a921ebcb6e696
Normal-3905.png,Normal,Normal/images/Normal-3905.png,Normal/masks/Normal-3905.png,True,G12836,test,f7718233692c34f9dbbdd8d3db56b2f1,9978c6871f168e1b
Normal-3906.png,Normal,Normal/images/Normal-3906.png,Normal/masks/Normal-3906.png,True,G12837,train,8d56d564a83c5cae65ec21710d8746c3,fd7cc30b1d080bba
Normal-3907.png,Normal,Normal/images/Normal-3907.png,Normal/masks/Normal-3907.png,True,G12838,train,dcbe9c4ec616332f8133bb435434a61a,91795e47091d9c4f
Normal-3908.png,Normal,Normal/images/Normal-3908.png,Normal/masks/Normal-3908.png,True,G12839,val,1adeaa5487da4fdd9bb41f92dcab7873,995f061a497a5a7a
Normal-3909.png,Normal,Normal/images/Normal-3909.png,Normal/masks/Normal-3909.png,True,G12840,val,47b0fbb4759301908cb46fb8c246aced,cc6ec70b5d0e0b0f
Normal-391.png,Normal,Normal/images/Normal-391.png,Normal/masks/Normal-391.png,True,G12841,test,ae05e10be806a7067e68eaa09eafafda,913dc31b98ce99b4
Normal-3910.png,Normal,Normal/images/Normal-3910.png,Normal/masks/Normal-3910.png,True,G12842,test,f61403e9574733ebecf0a697fd48092f,997f871ac2871a93
Normal-3911.png,Normal,Normal/images/Normal-3911.png,Normal/masks/Normal-3911.png,True,G12843,train,583a7accc57ff0d8853bee1fee49d99e,dd2dc60a5c6a39e2
Normal-3912.png,Normal,Normal/images/Normal-3912.png,Normal/masks/Normal-3912.png,True,G12844,train,72ef9c31702573c5b921ce0c5ba529ec,dc2ed14a0f4a0f4f
Normal-3913.png,Normal,Normal/images/Normal-3913.png,Normal/masks/Normal-3913.png,True,G12845,val,097dda4cecc43e2cf65fed0e32dd6c6c,997d061a986bb8e5
Normal-3914.png,Normal,Normal/images/Normal-3914.png,Normal/masks/Normal-3914.png,True,G12846,train,7cff12d495a2c9999c93d5b4399126ca,9178d656181d5f39
Normal-3915.png,Normal,Normal/images/Normal-3915.png,Normal/masks/Normal-3915.png,True,G12847,train,424809159c1c3f9bb714c7704dcefd80,c83c87071b571f1e
Normal-3916.png,Normal,Normal/images/Normal-3916.png,Normal/masks/Normal-3916.png,True,G12848,train,a1346583a74b4bad7f94273741b3631a,82631d1d1f78735a
Normal-3917.png,Normal,Normal/images/Normal-3917.png,Normal/masks/Normal-3917.png,True,G12849,test,b299131e123cc87dcb96441c25ef33b4,817a57961e196b9a
Normal-3918.png,Normal,Normal/images/Normal-3918.png,Normal/masks/Normal-3918.png,True,G12850,train,30200b9d89fd159d7d54cc39ae77428c,993d461a5d4b396a
Normal-3919.png,Normal,Normal/images/Normal-3919.png,Normal/masks/Normal-3919.png,True,G12851,train,573b3334c01de6333467d2cf0692655a,dd2ec24b5d4c0e1d
Normal-392.png,Normal,Normal/images/Normal-392.png,Normal/masks/Normal-392.png,True,G12852,train,11af1418de2533099f25aa103bcbc83e,817b461a594f393d
Normal-3920.png,Normal,Normal/images/Normal-3920.png,Normal/masks/Normal-3920.png,True,G12853,train,4d7a0eb1089e388d94e6bfeef97be2bd,8179c7969acc9e98
Normal-3921.png,Normal,Normal/images/Normal-3921.png,Normal/masks/Normal-3921.png,True,G12854,train,0ffcb25c995db2d6f00d1e24b9328535,dd4cc79a18490f3e
Normal-3922.png,Normal,Normal/images/Normal-3922.png,Normal/masks/Normal-3922.png,True,G12855,train,dd13df795072608d444cb10702459ba9,d42f470f1a4f4f18
Normal-3923.png,Normal,Normal/images/Normal-3923.png,Normal/masks/Normal-3923.png,True,G12856,train,48f3488bd970b4764742be4db09d9f2c,8979471e554f1d49
Normal-3924.png,Normal,Normal/images/Normal-3924.png,Normal/masks/Normal-3924.png,True,G12857,train,4b58d3f3ac5cf7ae7f2bb7175bdc06f5,d93ad61e181f1b0d
Normal-3925.png,Normal,Normal/images/Normal-3925.png,Normal/masks/Normal-3925.png,True,G12858,val,8009497ae7c711ac9fea1527b3255ec5,dc2c43434b4f4e4f
Normal-3926.png,Normal,Normal/images/Normal-3926.png,Normal/masks/Normal-3926.png,True,G12859,test,9e1b16fe4602423a1d9c48b3e1f09cda,dd2cc6521bc34a1f
Normal-3927.png,Normal,Normal/images/Normal-3927.png,Normal/masks/Normal-3927.png,True,G12860,train,3ea24de2f54061787ce58b8ac586055c,d52e424b0bcd2cfc
Normal-3928.png,Normal,Normal/images/Normal-3928.png,Normal/masks/Normal-3928.png,True,G12861,test,401cd042584d6069ef4828cfd41c9c6c,a93986961b5d5f18
Normal-3929.png,Normal,Normal/images/Normal-3929.png,Normal/masks/Normal-3929.png,True,G12862,train,3f263992a1ae0307e923eca468e27dbd,d42e435f5d0c0e5d
Normal-393.png,Normal,Normal/images/Normal-393.png,Normal/masks/Normal-393.png,True,G12863,train,5a4d87d00d3493f2dda5ff3ca62706d1,91fe8e3a8d29346c
Normal-3930.png,Normal,Normal/images/Normal-3930.png,Normal/masks/Normal-3930.png,True,G12864,train,7eef41656ce29b5148e65bb62202daf7,997847121b6d4f39
Normal-3931.png,Normal,Normal/images/Normal-3931.png,Normal/masks/Normal-3931.png,True,G12865,val,8b0b86120094ea283d907d779caefdf8,987945961a995dda
Normal-3932.png,Normal,Normal/images/Normal-3932.png,Normal/masks/Normal-3932.png,True,G12866,train,26f7ff91dd17b1afb24dc4976be24555,986d861ac05f7acd
Normal-3933.png,Normal,Normal/images/Normal-3933.png,Normal/masks/Normal-3933.png,True,G12867,train,4ca4c4b05304417eeb9a2263e854f564,b93d86571446531f
Normal-3934.png,Normal,Normal/images/Normal-3934.png,Normal/masks/Normal-3934.png,True,G12868,train,39e19e0d5a0a903224025f8c9f27867d,993f04d21b455eba
Normal-3935.png,Normal,Normal/images/Normal-3935.png,Normal/masks/Normal-3935.png,True,G12869,train,71a115209b07ae55756a74fd82907bff,d938d40a176b1ead
Normal-3936.png,Normal,Normal/images/Normal-3936.png,Normal/masks/Normal-3936.png,True,G12870,train,ba5c4218996c598ce97d4ea62e2557b8,8178165e79ade1b1
Normal-3937.png,Normal,Normal/images/Normal-3937.png,Normal/masks/Normal-3937.png,True,G12871,train,d3dee18bf914d96141586cc6d68771bf,9978471a1a4d5f39
Normal-3938.png,Normal,Normal/images/Normal-3938.png,Normal/masks/Normal-3938.png,True,G12872,train,8950520ce1d98c4ad1f653c8bdfa8116,897d871a194d78f4
Normal-3939.png,Normal,Normal/images/Normal-3939.png,Normal/masks/Normal-3939.png,True,G12873,train,be8e7381a06d364b36ad6052ce4a3961,882ac75f474f1a4d
Normal-394.png,Normal,Normal/images/Normal-394.png,Normal/masks/Normal-394.png,True,G12874,train,d0a2a6f5d594259e6c4a02f747598fc8,8168c65ad86d79d5
Normal-3940.png,Normal,Normal/images/Normal-3940.png,Normal/masks/Normal-3940.png,True,G12875,val,bb273af53739bf092365201768edea3a,dc2cc21f0a8f2df8
Normal-3941.png,Normal,Normal/images/Normal-3941.png,Normal/masks/Normal-3941.png,True,G12876,train,bbc6bb11821f00d2eb7382422919ac43,913f464b094d6ff0
Normal-3942.png,Normal,Normal/images/Normal-3942.png,Normal/masks/Normal-3942.png,True,G12877,val,c7fb5f9b99fc74c5687c24f1b365ce37,dd38c31fd0071b93
Normal-3943.png,Normal,Normal/images/Normal-3943.png,Normal/masks/Normal-3943.png,True,G12878,test,6cece664ff88ae7a0ecced8957f63492,817b5616174e7e15
Normal-3944.png,Normal,Normal/images/Normal-3944.png,Normal/masks/Normal-3944.png,True,G12879,train,e2eb2d5e13fa9e4fe46e8928289d0849,dd2f02514f49b94d
Normal-3945.png,Normal,Normal/images/Normal-3945.png,Normal/masks/Normal-3945.png,True,G12880,test,33d4edcc25b38830df95ff1c875a46e9,957807141f1f1d1f
Normal-3946.png,Normal,Normal/images/Normal-3946.png,Normal/masks/Normal-3946.png,True,G12881,test,612c21d8f13e5ace71df63e4bce12b68,916ec61a4bc71a5b
Normal-3947.png,Normal,Normal/images/Normal-3947.png,Normal/masks/Normal-3947.png,True,G12882,train,bf170ffb18d6b3fee06a520778cb4517,9919961a59fd78a8
Normal-3948.png,Normal,Normal/images/Normal-3948.png,Normal/masks/Normal-3948.png,True,G12883,train,1ba36628e96c69266cddc9719d8738fe,d93c565b54471d58
Normal-3949.png,Normal,Normal/images/Normal-3949.png,Normal/masks/Normal-3949.png,True,G12884,val,957198230786b9c814b8acd19fe3dafb,dd2fc01b1f4e0a9c
Normal-395.png,Normal,Normal/images/Normal-395.png,Normal/masks/Normal-395.png,True,G12885,train,c04ba46c55fada5d0e6c236f23761e1e,9139c61af9493d78
Normal-3950.png,Normal,Normal/images/Normal-3950.png,Normal/masks/Normal-3950.png,True,G12886,train,b8380804d2f4f38cc13bae433d99dfe4,dc3c0303594f6e7c
Normal-3951.png,Normal,Normal/images/Normal-3951.png,Normal/masks/Normal-3951.png,True,G12887,train,54bfa7d7739f660229d228c9a5d02542,d93dc54716434f16
Normal-3952.png,Normal,Normal/images/Normal-3952.png,Normal/masks/Normal-3952.png,True,G12888,train,8cd4fd581bcf76f7550ca214cc46f8c2,fd6cc21b1d460b39
Normal-3953.png,Normal,Normal/images/Normal-3953.png,Normal/masks/Normal-3953.png,True,G12889,train,31df58adb3a3e2292b78804baff205ab,d83d420f4c4b3d5e
Normal-3954.png,Normal,Normal/images/Normal-3954.png,Normal/masks/Normal-3954.png,True,G12890,train,adc318a555963c647e6be8f0b337d4c4,99689e1a9feca8a1
Normal-3955.png,Normal,Normal/images/Normal-3955.png,Normal/masks/Normal-3955.png,True,G12891,train,4b8072c853fd17f5c3a01948c8156d4d,817f070a1f555f58
Normal-3956.png,Normal,Normal/images/Normal-3956.png,Normal/masks/Normal-3956.png,True,G12892,train,5b82f028480078a0664c6a30485974b9,dd3cc24f0e4c1bca
Normal-3957.png,Normal,Normal/images/Normal-3957.png,Normal/masks/Normal-3957.png,True,G12893,train,59e03977d719c98abfce6744fb66e0cd,dc2a530e4c5b7969
Normal-3958.png,Normal,Normal/images/Normal-3958.png,Normal/masks/Normal-3958.png,True,G12894,train,9f514a48c8ec9812e67fbcb80245a757,dc3dc3031b8e4e59
Normal-3959.png,Normal,Normal/images/Normal-3959.png,Normal/masks/Normal-3959.png,True,G12895,train,5cad33079091af3cb3a26652ea0902f3,d43d42575f060f1d
Normal-396.png,Normal,Normal/images/Normal-396.png,Normal/masks/Normal-396.png,True,G12896,train,f49326a21db417eaf53d3279eed235a3,957c873a9b0d38b4
Normal-3960.png,Normal,Normal/images/Normal-3960.png,Normal/masks/Normal-3960.png,True,G12897,val,03b0e3409338afdaba58246b67031c69,805d161b1b0f1f6f
Normal-3961.png,Normal,Normal/images/Normal-3961.png,Normal/masks/Normal-3961.png,True,G12898,val,726ce847515a355594ba03729f259a3d,992d861a1d4f3ab9
Normal-3962.png,Normal,Normal/images/Normal-3962.png,Normal/masks/Normal-3962.png,True,G12899,val,254e9a89a8ae4c3ca87299918690d0e5,9c2e035b0b570f5d
Normal-3963.png,Normal,Normal/images/Normal-3963.png,Normal/masks/Normal-3963.png,True,G12900,test,d013f3d48ca3df00c2aa7f12947ce36d,d9a98698868f3e87
Normal-3964.png,Normal,Normal/images/Normal-3964.png,Normal/masks/Normal-3964.png,True,G12901,test,07221c1d58a733d3044eeca3504a7d49,dd2c56431f424b1f
Normal-3965.png,Normal,Normal/images/Normal-3965.png,Normal/masks/Normal-3965.png,True,G12902,test,a4027076f0f1b76e015cb92f7245ac38,9d2c401ac54f3f6d
Normal-3966.png,Normal,Normal/images/Normal-3966.png,Normal/masks/Normal-3966.png,True,G12903,train,921efe9565557f61a23e910ed83b3304,dc2cc31b5d6a187a
Normal-3967.png,Normal,Normal/images/Normal-3967.png,Normal/masks/Normal-3967.png,True,G12904,train,265248d507bc71d623f86441a13a555e,f8298516865f1e4f
Normal-3968.png,Normal,Normal/images/Normal-3968.png,Normal/masks/Normal-3968.png,True,G12905,train,c7a4b7352aac87d81758b60e07deacab,887f065a1f6c78e1
Normal-3969.png,Normal,Normal/images/Normal-3969.png,Normal/masks/Normal-3969.png,True,G12906,val,0daed7331aa841425f08599cbb33469b,987f061e1a5f5958
Normal-397.png,Normal,Normal/images/Normal-397.png,Normal/masks/Normal-397.png,True,G12907,train,c7cf7092baef4ea1c8be5f748e3b49aa,916fc63ae92929c5
Normal-3970.png,Normal,Normal/images/Normal-3970.png,Normal/masks/Normal-3970.png,True,G12908,train,f244df9c21f6c7e4ea0ea0db3ce5c756,9939061a59fa79e8
Normal-3971.png,Normal,Normal/images/Normal-3971.png,Normal/masks/Normal-3971.png,True,G12909,train,998cb2d5499283e6f2e8106035c63440,d969845b954f1f0c
Normal-3972.png,Normal,Normal/images/Normal-3972.png,Normal/masks/Normal-3972.png,True,G12910,train,cbe3f046d10ac3500758022ebbb69854,d968d21b5e471959
Normal-3973.png,Normal,Normal/images/Normal-3973.png,Normal/masks/Normal-3973.png,True,G12911,train,a7d790096c0329118cd280f65f165dca,dd3dc21ed0471e49
Normal-3974.png,Normal,Normal/images/Normal-3974.png,Normal/masks/Normal-3974.png,True,G12912,train,73e2ec6957e8841d74bf01373a089f49,dc2dc25f1f023f18
Normal-3975.png,Normal,Normal/images/Normal-3975.png,Normal/masks/Normal-3975.png,True,G12913,train,60245fa62ebc1339a51932ceb5dc7bdd,ed0cc60f4ccc1dad
Normal-3976.png,Normal,Normal/images/Normal-3976.png,Normal/masks/Normal-3976.png,True,G12914,train,b6163522e3859c36c7266f94625a462c,993dc61a1c471e3d
Normal-3977.png,Normal,Normal/images/Normal-3977.png,Normal/masks/Normal-3977.png,True,G12915,train,6e54fc1c75fb070819adb376cdc94b6e,8d6c831b4d1e784f
Normal-3978.png,Normal,Normal/images/Normal-3978.png,Normal/masks/Normal-3978.png,True,G12916,train,3d31314ac338672d6c765f18b1f6966a,d93dc61b1d4b1d48
Normal-3979.png,Normal,Normal/images/Normal-3979.png,Normal/masks/Normal-3979.png,True,G12917,test,e44cb5a0b9ced6d41aa8079b922b381a,d42d43431d4a6f3d
Normal-398.png,Normal,Normal/images/Normal-398.png,Normal/masks/Normal-398.png,True,G12918,val,ef1d8012e5190d616fb9d62bf12f8e08,81a9c63ab9693979
Normal-3980.png,Normal,Normal/images/Normal-3980.png,Normal/masks/Normal-3980.png,True,G12919,train,8b7c627d080a69fb00f2db0a7e50a68d,9969041e155e1f5f
Normal-3981.png,Normal,Normal/images/Normal-3981.png,Normal/masks/Normal-3981.png,True,G12920,train,8f518c6185336b59f0c1fbe3dc69cd43,897b061a1e557b39
Normal-3982.png,Normal,Normal/images/Normal-3982.png,Normal/masks/Normal-3982.png,True,G12921,test,a72497066880389311474a24cf40b028,8178061a5f5d5b75
Normal-3983.png,Normal,Normal/images/Normal-3983.png,Normal/masks/Normal-3983.png,True,G12922,train,408d5236af76572cd5b8da87a3ad0741,9979d71e541f180b
Normal-3984.png,Normal,Normal/images/Normal-3984.png,Normal/masks/Normal-3984.png,True,G12923,train,235627ab96aabb4752d0521da03f7258,9968071e5e5b5a78
Normal-3985.png,Normal,Normal/images/Normal-3985.png,Normal/masks/Normal-3985.png,True,G12924,train,33bc0d977eda916853e2e88a3e0170d8,917b47161a4f5f18
Normal-3986.png,Normal,Normal/images/Normal-3986.png,Normal/masks/Normal-3986.png,True,G12925,val,dd7073987eaa059733593d92e71272c3,997dc6121d4f1959
Normal-3987.png,Normal,Normal/images/Normal-3987.png,Normal/masks/Normal-3987.png,True,G12926,test,d29aa797b42a084ec0582157499d9956,dd2cc70a5d0c19bd
Normal-3988.png,Normal,Normal/images/Normal-3988.png,Normal/masks/Normal-3988.png,True,G12927,test,74b0eccfd2316615e207f7e8d3d76ae4,9169865a1b4e697b
Normal-3989.png,Normal,Normal/images/Normal-3989.png,Normal/masks/Normal-3989.png,True,G12928,test,72e93d23150e83c132dd2868f0c07a2a,dd2ac31f486d1e58
Normal-399.png,Normal,Normal/images/Normal-399.png,Normal/masks/Normal-399.png,True,G12929,train,affd19e50c3309322cfda076ebb3bc72,d06ec77d8d452cc4
Normal-3990.png,Normal,Normal/images/Normal-3990.png,Normal/masks/Normal-3990.png,True,G12930,train,de27e2b343aa92a80027c906a6dc45c9,d03d435c1b0e2dbd
Normal-3991.png,Normal,Normal/images/Normal-3991.png,Normal/masks/Normal-3991.png,True,G12931,train,a6aa0808fef1266f0869515b21ff6b32,997dc45a564e5e18
Normal-3992.png,Normal,Normal/images/Normal-3992.png,Normal/masks/Normal-3992.png,True,G12932,train,8860af9cdb116f40390ff0a7cdee1154,887d4587165b4f47
Normal-3993.png,Normal,Normal/images/Normal-3993.png,Normal/masks/Normal-3993.png,True,G12933,train,f53c062f00c68f527ee4aaa077a6e2b2,d939d61f144a1b6a
Normal-3994.png,Normal,Normal/images/Normal-3994.png,Normal/masks/Normal-3994.png,True,G12934,train,a4cc976291c186c98c09dba351e55071,893ec64b1c5e6b38
Normal-3995.png,Normal,Normal/images/Normal-3995.png,Normal/masks/Normal-3995.png,True,G12935,train,bce4b76320e50bade2d499347901431d,fd28861f051e4d4f
Normal-3996.png,Normal,Normal/images/Normal-3996.png,Normal/masks/Normal-3996.png,True,G12936,train,61c507d4e6f90344234af83bafc25396,8979061e1a4d7f39
Normal-3997.png,Normal,Normal/images/Normal-3997.png,Normal/masks/Normal-3997.png,True,G12937,train,8931566d78fcc146e67a67f25515a2f5,dc3dc30b484f2f98
Normal-3998.png,Normal,Normal/images/Normal-3998.png,Normal/masks/Normal-3998.png,True,G12938,train,2e682dc3d215487fd7e5e2d1f0daf547,c83d421b1b4f3c79
Normal-3999.png,Normal,Normal/images/Normal-3999.png,Normal/masks/Normal-3999.png,True,G12939,train,f620253300e0a3fd0a80bf8757c10f7f,9361c61e1f5f5838
Normal-4.png,Normal,Normal/images/Normal-4.png,Normal/masks/Normal-4.png,True,G12940,val,0bb5e02779d259ca36f25e2c1e46025b,dd2ec21b01c71b57
Normal-40.png,Normal,Normal/images/Normal-40.png,Normal/masks/Normal-40.png,True,G12941,val,89572698250451e46198aedbf78e03cb,d12b570a5f035f1c
Normal-400.png,Normal,Normal/images/Normal-400.png,Normal/masks/Normal-400.png,True,G12942,val,ef88276f9bdcb067933a680431669bcf,907cc73aba2d38a5
Normal-4000.png,Normal,Normal/images/Normal-4000.png,Normal/masks/Normal-4000.png,True,G12943,test,87b1dc957e6dfe6d0ece131523e14eef,b97e861b094f18ba
Normal-4001.png,Normal,Normal/images/Normal-4001.png,Normal/masks/Normal-4001.png,True,G12944,train,1ff58269acc43a42eef6b331e4823f45,dd2cc30b4dac18de
Normal-4002.png,Normal,Normal/images/Normal-4002.png,Normal/masks/Normal-4002.png,True,G12945,train,33f8bc5106f3d89f8b0d299b2639445b,d1348e87191bce3e
Normal-4003.png,Normal,Normal/images/Normal-4003.png,Normal/masks/Normal-4003.png,True,G12946,train,8076147b3793da130ad8031f0cfaefb1,885d579b5fa07aa0
Normal-4004.png,Normal,Normal/images/Normal-4004.png,Normal/masks/Normal-4004.png,True,G12947,train,1106fe511df36996919af4835a358a8e,9179afc6161ac54e
Normal-4005.png,Normal,Normal/images/Normal-4005.png,Normal/masks/Normal-4005.png,True,G12948,train,870169206c157c3eca83ad0ba75c9b01,c9301b1e39cf2db1
Normal-4006.png,Normal,Normal/images/Normal-4006.png,Normal/masks/Normal-4006.png,True,G12949,val,f209d8733e91d4b69597c2607b103cf2,dc2ed34b0ec24b0f
Normal-4007.png,Normal,Normal/images/Normal-4007.png,Normal/masks/Normal-4007.png,True,G12950,train,48c6a56b6a7fa13b9136306dc5bc5992,dd2c821bc54e9b4e
Normal-4008.png,Normal,Normal/images/Normal-4008.png,Normal/masks/Normal-4008.png,True,G12951,train,544ee7687e1017afee227387bf1dfc7c,dd0a430d2c2dbcf6
Normal-4009.png,Normal,Normal/images/Normal-4009.png,Normal/masks/Normal-4009.png,True,G12952,train,0b0d904a42ddad9cbadcdefe802927b6,f869871a1e4c1f1b
Normal-401.png,Normal,Normal/images/Normal-401.png,Normal/masks/Normal-401.png,True,G12953,test,806b24da3b1b1a7527746c3b048e05ae,907de31a994d3c9c
Normal-4010.png,Normal,Normal/images/Normal-4010.png,Normal/masks/Normal-4010.png,True,G12954,train,82d8af58c932063f98ab1fed05d7fa5c,dd2c02c30b7c4a7f
Normal-4011.png,Normal,Normal/images/Normal-4011.png,Normal/masks/Normal-4011.png,True,G12955,train,1455e54b919a057a21225edc24d47f5b,c03cdb43d10fae0f
Normal-4012.png,Normal,Normal/images/Normal-4012.png,Normal/masks/Normal-4012.png,True,G12956,train,1b59e76753a1dfe85e11fdf870b534c4,986dc61ac55b394d
Normal-4013.png,Normal,Normal/images/Normal-4013.png,Normal/masks/Normal-4013.png,True,G12957,val,62f5963faf00bd17b59ed5492a11a9bc,dd2dc20bc54a1dec
Normal-4014.png,Normal,Normal/images/Normal-4014.png,Normal/masks/Normal-4014.png,True,G12958,val,8c0e5f4d5cbedfafede0932af51f2578,d42c42535b474f5b
Normal-4015.png,Normal,Normal/images/Normal-4015.png,Normal/masks/Normal-4015.png,True,G12959,train,a207531c402fc1b5e425e42517d4a676,c93905065a1d5f7b
Normal-4016.png,Normal,Normal/images/Normal-4016.png,Normal/masks/Normal-4016.png,True,G12960,val,1be45417a176b2ddc8a7726801e4ee0d,c13946071b17cf1f
Normal-4017.png,Normal,Normal/images/Normal-4017.png,Normal/masks/Normal-4017.png,True,G12961,test,71cefb841b08976a896afc9aa670dad4,817d06121b5f787b
Normal-4018.png,Normal,Normal/images/Normal-4018.png,Normal/masks/Normal-4018.png,True,G12962,train,36182af6e6b2ca024c6ebe1f29c7500b,9168065a1b4f6f3b
Normal-4019.png,Normal,Normal/images/Normal-4019.png,Normal/masks/Normal-4019.png,True,G12963,train,27afec800ce5ff93358268adca2ce256,d83b57861e192a5e
Normal-402.png,Normal,Normal/images/Normal-402.png,Normal/masks/Normal-402.png,True,G12964,train,20400c3c3439a5e204ff60464baff43d,c0acc65ed64f5c43
Normal-4020.png,Normal,Normal/images/Normal-4020.png,Normal/masks/Normal-4020.png,True,G12965,train,20349410cad459cdf22c7a293fe4e55a,c139061f594a7d79
Normal-4021.png,Normal,Normal/images/Normal-4021.png,Normal/masks/Normal-4021.png,True,G12966,test,5c23784114626354dca1a5701876e93b,9968c65759865a5d
Normal-4022.png,Normal,Normal/images/Normal-4022.png,Normal/masks/Normal-4022.png,True,G12967,train,327851359970837d84d482e8546bb285,913fc700ba566d56
Normal-4023.png,Normal,Normal/images/Normal-4023.png,Normal/masks/Normal-4023.png,True,G12968,train,8f1b9397dfe59ae87a0b6131ba0b1592,997f861a195d5938
Normal-4024.png,Normal,Normal/images/Normal-4024.png,Normal/masks/Normal-4024.png,True,G12969,train,6926a001a59126d6dfaa1f7fc9ad9cb4,997805161f945f3b
Normal-4025.png,Normal,Normal/images/Normal-4025.png,Normal/masks/Normal-4025.png,True,G12970,test,dce5b25cb10bb67c71f0c708e2b21278,996d46c21b1b4f3c
Normal-4026.png,Normal,Normal/images/Normal-4026.png,Normal/masks/Normal-4026.png,True,G12971,train,58d010018620d970b397b7f7fb97ae15,d97b851b1f441e1c
Normal-4027.png,Normal,Normal/images/Normal-4027.png,Normal/masks/Normal-4027.png,True,G12972,train,7c1248686b3d264062dcabf7fba5c504,817bef045e0a7b19
Normal-4028.png,Normal,Normal/images/Normal-4028.png,Normal/masks/Normal-4028.png,True,G12973,train,ea4179e550cd96046ba65e27067035bb,dd2ec253454e3e2a
Normal-4029.png,Normal,Normal/images/Normal-4029.png,Normal/masks/Normal-4029.png,True,G12974,test,a63fef72485046455d815127dab332f7,dd0a431f4d0e5d4b
Normal-403.png,Normal,Normal/images/Normal-403.png,Normal/masks/Normal-403.png,True,G12975,train,646d4ed00b276ab69787645038113a1d,912bc21a9b78397d
Normal-4030.png,Normal,Normal/images/Normal-4030.png,Normal/masks/Normal-4030.png,True,G12976,train,320de8a30ceb167f14e83bbee4d0eb18,dd7dc5921a094d1e
Normal-4031.png,Normal,Normal/images/Normal-4031.png,Normal/masks/Normal-4031.png,True,G12977,train,00036e95d11e9bf3c7e504477557b375,8075425f5a5d5d59
Normal-4032.png,Normal,Normal/images/Normal-4032.png,Normal/masks/Normal-4032.png,True,G12978,train,642aff0bf6803aebb172614f0c0f7424,dd3d42530b424f1f
Normal-4033.png,Normal,Normal/images/Normal-4033.png,Normal/masks/Normal-4033.png,True,G12979,train,b9a87abe93a1937684b209689fcbed64,9168861e5bcf7878
Normal-4034.png,Normal,Normal/images/Normal-4034.png,Normal/masks/Normal-4034.png,True,G12980,val,35590ac1d0748be2f562b8c00059c688,95696c0e1b799cc9
Normal-4035.png,Normal,Normal/images/Normal-4035.png,Normal/masks/Normal-4035.png,True,G12981,val,27848321f6eb3be5f90359bb48fe8a6e,d83f46431b0f4e1b
Normal-4036.png,Normal,Normal/images/Normal-4036.png,Normal/masks/Normal-4036.png,True,G12982,val,ce46928f3ed18962fb7a4f3b54b380b4,dc2cc30b444b7d6e
Normal-4037.png,Normal,Normal/images/Normal-4037.png,Normal/masks/Normal-4037.png,True,G12983,train,81fd80ca52a36bb8655499f761523025,9978c71a1b4f1e49
Normal-4038.png,Normal,Normal/images/Normal-4038.png,Normal/masks/Normal-4038.png,True,G12984,test,0195bd611b33c99f6b68dc207b8d6a32,dc2cd2434f4e4b0f
Normal-4039.png,Normal,Normal/images/Normal-4039.png,Normal/masks/Normal-4039.png,True,G12985,train,80f58ea44f539bc80c58553acd992fd9,917856065a5f6979
Normal-404.png,Normal,Normal/images/Normal-404.png,Normal/masks/Normal-404.png,True,G12986,train,50fce92a0fe39574080af8200dc99e81,812bc65af9693969
Normal-4040.png,Normal,Normal/images/Normal-4040.png,Normal/masks/Normal-4040.png,True,G12987,train,892ce4dbf8889c1ab2bb53537205f783,dd3dc0121bc34f1d
Normal-4041.png,Normal,Normal/images/Normal-4041.png,Normal/masks/Normal-4041.png,True,G12988,train,dd4282cd24edff2657de9a39c8610b59,d92f861a0d50797d
Normal-4042.png,Normal,Normal/images/Normal-4042.png,Normal/masks/Normal-4042.png,True,G12989,train,6ca05cb69a25679c03faeae3618e7edd,dc2d46c30b4eee1a
Normal-4043.png,Normal,Normal/images/Normal-4043.png,Normal/masks/Normal-4043.png,True,G12990,train,36a2cc290839aad080c00c71de7937a8,d93d861b1c4f391c
Normal-4044.png,Normal,Normal/images/Normal-4044.png,Normal/masks/Normal-4044.png,True,G12991,train,6f479267c421bfd5627ae324f75374df,dc3d434b5d4a6c2c
Normal-4045.png,Normal,Normal/images/Normal-4045.png,Normal/masks/Normal-4045.png,True,G12992,train,69bdd2abdf62efd7d9861b38ffec46d8,dd2ad20b1f441b5d
Normal-4046.png,Normal,Normal/images/Normal-4046.png,Normal/masks/Normal-4046.png,True,G12993,train,965e5110536217367893a5fe0ffc4eae,d93dc61f1a451f18
Normal-4047.png,Normal,Normal/images/Normal-4047.png,Normal/masks/Normal-4047.png,True,G12994,train,d345207e4f80a69f0018be134d8d2284,dd2cc31f0c4f0d4b
Normal-4048.png,Normal,Normal/images/Normal-4048.png,Normal/masks/Normal-4048.png,True,G12995,train,2bea7e052d79c7b427b085dfaa002fe0,997dc61bd50f1541
Normal-4049.png,Normal,Normal/images/Normal-4049.png,Normal/masks/Normal-4049.png,True,G12996,train,62c0f2ed32f2f7cb97716ffe9412b374,813b16181d5f7979
Normal-405.png,Normal,Normal/images/Normal-405.png,Normal/masks/Normal-405.png,True,G12997,train,c969e75200aa8f97c4c9873df1da9dfe,d07ede5b9a8a2898
Normal-4050.png,Normal,Normal/images/Normal-4050.png,Normal/masks/Normal-4050.png,True,G12998,train,ffd6f950a73d5573e6cbced570ef3a04,cc35430b4e2f6c6e
Normal-4051.png,Normal,Normal/images/Normal-4051.png,Normal/masks/Normal-4051.png,True,G12999,test,1d48ccb725db4768a8b441bb79890dae,dc2e431b424f1e4f
Normal-4052.png,Normal,Normal/images/Normal-4052.png,Normal/masks/Normal-4052.png,True,G13000,val,6f63ee7d9b251a86cdc61c7d2c72c85e,8879871e5a0f595b
Normal-4053.png,Normal,Normal/images/Normal-4053.png,Normal/masks/Normal-4053.png,True,G13001,val,e29fcaa7b2c8b2370dd2c9572ec66f81,d42f42575d4f0e1c
Normal-4054.png,Normal,Normal/images/Normal-4054.png,Normal/masks/Normal-4054.png,True,G13002,val,cdb723e466893c9ebfadb5a41d77f8d1,9978161a1ae4eab7
Normal-4055.png,Normal,Normal/images/Normal-4055.png,Normal/masks/Normal-4055.png,True,G13003,test,47e6ec5cfad2de4dfa56a9fb75022ee3,d43d03031f4e2f3d
Normal-4056.png,Normal,Normal/images/Normal-4056.png,Normal/masks/Normal-4056.png,True,G13004,test,220f2a83db0562981efa0690ed8a4e0c,993b861a585f7859
Normal-4057.png,Normal,Normal/images/Normal-4057.png,Normal/masks/Normal-4057.png,True,G13005,train,448bf18fc349fe94ae0742192e227c1b,997bc51e1a075a1b
Normal-4058.png,Normal,Normal/images/Normal-4058.png,Normal/masks/Normal-4058.png,True,G13006,train,e438c54a6589b57546cdb2ffdb67e111,91691c5b7b6963c2
Normal-4059.png,Normal,Normal/images/Normal-4059.png,Normal/masks/Normal-4059.png,True,G13007,train,73919d1486e5fe13a772dd17216fdf5b,c9344e0e39ce6d39
Normal-406.png,Normal,Normal/images/Normal-406.png,Normal/masks/Normal-406.png,True,G13008,train,9129a49742cbcdab54913c6df012e495,806dc71aeb6968e5
Normal-4060.png,Normal,Normal/images/Normal-4060.png,Normal/masks/Normal-4060.png,True,G13009,train,7ba614714d0c46897099f73e4f6df844,917b461a3b293d35
Normal-4061.png,Normal,Normal/images/Normal-4061.png,Normal/masks/Normal-4061.png,True,G13010,train,5d285c161296abb2b89f5c4fc2afb5d3,c87d421a1b6d3cf1
Normal-4062.png,Normal,Normal/images/Normal-4062.png,Normal/masks/Normal-4062.png,True,G13011,val,c5b4a941d61c1e2d54b55b39a40e8fb9,d97ad64b584c6954
Normal-4063.png,Normal,Normal/images/Normal-4063.png,Normal/masks/Normal-4063.png,True,G13012,train,3aabbeeabad202ae74c3947606e17909,9979c6065a15fe19
Normal-4064.png,Normal,Normal/images/Normal-4064.png,Normal/masks/Normal-4064.png,True,G13013,train,c9e354ad6d7a0e6afa0cd17b3b5d779e,dd2a531b0a1f0b2f
Normal-4065.png,Normal,Normal/images/Normal-4065.png,Normal/masks/Normal-4065.png,True,G13014,train,9ae7b4333b1ebdf66d0cf5aac6eb5ca4,9879c55e5a485b39
Normal-4066.png,Normal,Normal/images/Normal-4066.png,Normal/masks/Normal-4066.png,True,G13015,train,658b105ffbaa8721d844a9e1e3920ffe,9d2e425b69c47ae8
Normal-4067.png,Normal,Normal/images/Normal-4067.png,Normal/masks/Normal-4067.png,True,G13016,train,019f68c1f99b1ab3c19c278810a3d8d9,89795656185d4f1b
Normal-4068.png,Normal,Normal/images/Normal-4068.png,Normal/masks/Normal-4068.png,True,G13017,train,b088af70f798cf530a16f5770e214ad0,d0bdc21d164f1f1a
Normal-4069.png,Normal,Normal/images/Normal-4069.png,Normal/masks/Normal-4069.png,True,G13018,train,86d536dc33dad06895eb3d3c5b5e3fc5,997b861a191b1b3e
Normal-407.png,Normal,Normal/images/Normal-407.png,Normal/masks/Normal-407.png,True,G13019,train,28ab9d5ab1e065b5bbd78e487bd9bda6,d02fc71a99493cc7
Normal-4070.png,Normal,Normal/images/Normal-4070.png,Normal/masks/Normal-4070.png,True,G13020,val,896756eab7bbf276bfde5ba2fb1c91f9,807a47161a5d797b
Normal-4071.png,Normal,Normal/images/Normal-4071.png,Normal/masks/Normal-4071.png,True,G13021,val,90bf0120f165378bdbfec1171a049d73,c0bfc209c64f7d45
Normal-4072.png,Normal,Normal/images/Normal-4072.png,Normal/masks/Normal-4072.png,True,G13022,test,f797291658925ba426a0d1ec37558b6b,997f065a1b4e3839
Normal-4073.png,Normal,Normal/images/Normal-4073.png,Normal/masks/Normal-4073.png,True,G13023,test,ae2dee28c0ae26ac6ffe130a1de60d60,d52c07071f0f4f4d
Normal-4074.png,Normal,Normal/images/Normal-4074.png,Normal/masks/Normal-4074.png,True,G13024,test,f4fe943c569f9fc8571aa2a76150fba4,c03cc70f1b1e4f3c
Normal-4075.png,Normal,Normal/images/Normal-4075.png,Normal/masks/Normal-4075.png,True,G13025,val,58a47641810f6c0590c2db39ca7bb421,99a80e1ad7787a4d
Normal-4076.png,Normal,Normal/images/Normal-4076.png,Normal/masks/Normal-4076.png,True,G13026,val,2b92c8fa7ac32bfbfe5ceb7510015f9e,997d065a954e1b47
Normal-4077.png,Normal,Normal/images/Normal-4077.png,Normal/masks/Normal-4077.png,True,G13027,test,b277293c48c3d962640f978d4237219b,93401d151f7b735a
Normal-4078.png,Normal,Normal/images/Normal-4078.png,Normal/masks/Normal-4078.png,True,G13028,train,d7541e97ef5b022253c01112a882bfc3,dc28d30bd40e1fe5
Normal-4079.png,Normal,Normal/images/Normal-4079.png,Normal/masks/Normal-4079.png,True,G13029,train,64ad2af1277f5e05be3a0a9b6ce2f19a,813f821b1ffeb8a0
Normal-408.png,Normal,Normal/images/Normal-408.png,Normal/masks/Normal-408.png,True,G13030,train,30fcc07833491000003a52203aab978c,917cc638ad4d3ccc
Normal-4080.png,Normal,Normal/images/Normal-4080.png,Normal/masks/Normal-4080.png,True,G13031,train,de8ec36c752a7969a8e38ff33d986908,d03dc20b08cf7f3c
Normal-4081.png,Normal,Normal/images/Normal-4081.png,Normal/masks/Normal-4081.png,True,G13032,train,43dd06c8756e7b13948ca7240e3383e8,997c8712154f5ae9
Normal-4082.png,Normal,Normal/images/Normal-4082.png,Normal/masks/Normal-4082.png,True,G13033,test,f38f1d4b3436e1873a838688dcf2e3a8,d12b465e18156f3d
Normal-4083.png,Normal,Normal/images/Normal-4083.png,Normal/masks/Normal-4083.png,True,G13034,train,f7c12d57edfdcc5c7b5e4a010b0e66a5,dc2dc21f1d440f5d
Normal-4084.png,Normal,Normal/images/Normal-4084.png,Normal/masks/Normal-4084.png,True,G13035,train,4a6379c90b70922cdde66e54c69dcd3e,d43d46411b2ced7c
Normal-4085.png,Normal,Normal/images/Normal-4085.png,Normal/masks/Normal-4085.png,True,G13036,train,922e331cacaf6a5183f1688810dcf101,dc3c4648da6e38ec
Normal-4086.png,Normal,Normal/images/Normal-4086.png,Normal/masks/Normal-4086.png,True,G13037,val,ab6ec0a136979094f08bd982a7759cbc,d13f421d184f1d79
Normal-4087.png,Normal,Normal/images/Normal-4087.png,Normal/masks/Normal-4087.png,True,G13038,train,16fc22368fc9ecd5fa6c9f6fc19295ce,882dd71f4d5a5cc1
Normal-4088.png,Normal,Normal/images/Normal-4088.png,Normal/masks/Normal-4088.png,True,G13039,train,317d65ce8f53ca7a2aec6b5e785d5c7a,91281e1a5b5b7979
Normal-4089.png,Normal,Normal/images/Normal-4089.png,Normal/masks/Normal-4089.png,True,G13040,test,b5c909e24856f9daef1c7bb925cdb736,817a16181b7b792f
Normal-409.png,Normal,Normal/images/Normal-409.png,Normal/masks/Normal-409.png,True,G13041,train,723c2e93c2030c31624aca4249cde9ce,9174861acd3d69bc
Normal-4090.png,Normal,Normal/images/Normal-4090.png,Normal/masks/Normal-4090.png,True,G13042,train,93d07583e4768d259e2efdd43af5c740,c13d4e461b18ee6d
Normal-4091.png,Normal,Normal/images/Normal-4091.png,Normal/masks/Normal-4091.png,True,G13043,train,199eec90aaa9a24ef4b0672e019f52eb,91699618195b795f
Normal-4092.png,Normal,Normal/images/Normal-4092.png,Normal/masks/Normal-4092.png,True,G13044,train,fc4b04f48599b8293d1c66a437b50916,d62d414d4e0f2f2e
Normal-4093.png,Normal,Normal/images/Normal-4093.png,Normal/masks/Normal-4093.png,True,G13045,train,b3d3350137f577392c17d2007f22c720,c17b56121a1f1f39
Normal-4094.png,Normal,Normal/images/Normal-4094.png,Normal/masks/Normal-4094.png,True,G13046,train,bb19a729d2702449b62ed45ed80d7ffa,891e471a5d4d5ce9
Normal-4095.png,Normal,Normal/images/Normal-4095.png,Normal/masks/Normal-4095.png,True,G13047,test,e71d456f2c246405a512f43446a89dd2,dc3dc25359c14b3c
Normal-4096.png,Normal,Normal/images/Normal-4096.png,Normal/masks/Normal-4096.png,True,G13048,train,8dbf7d2f8795ea418ab51d0d0323db09,9979921a4d0e3ee5
Normal-4097.png,Normal,Normal/images/Normal-4097.png,Normal/masks/Normal-4097.png,True,G13049,train,995f255d569fbace7a6da0380be9550f,9938061b5bee78b1
Normal-4098.png,Normal,Normal/images/Normal-4098.png,Normal/masks/Normal-4098.png,True,G13050,train,94321dc4a232ccc3780caeaac77a36b3,9139461e974e1c5f
Normal-4099.png,Normal,Normal/images/Normal-4099.png,Normal/masks/Normal-4099.png,True,G13051,train,2726bcb19419b3c8a0afcdc66d95b728,893c861a1fc51f5b
Normal-41.png,Normal,Normal/images/Normal-41.png,Normal/masks/Normal-41.png,True,G13052,train,093a78c636e9a080a838a451a0187351,d93d06531b4c4e7a
Normal-410.png,Normal,Normal/images/Normal-410.png,Normal/masks/Normal-410.png,True,G13053,train,9664f9f5345c88b5846876700a76e9d1,d13ac71af98c3ac4
Normal-4100.png,Normal,Normal/images/Normal-4100.png,Normal/masks/Normal-4100.png,True,G13054,train,844ec4b91b1e90ea1cbf6de3a2745809,81795649191fed39
Normal-4101.png,Normal,Normal/images/Normal-4101.png,Normal/masks/Normal-4101.png,True,G13055,train,f14576f48ae23f5d72fe75cb7082cc01,897906d61a195f6b
Normal-4102.png,Normal,Normal/images/Normal-4102.png,Normal/masks/Normal-4102.png,True,G13056,train,79f4998d3bd2bce50276e27ed86d4488,dc2d428f1f04ad3d
Normal-4103.png,Normal,Normal/images/Normal-4103.png,Normal/masks/Normal-4103.png,True,G13057,train,87b42bd0f4fbd31ac8117286255e8120,c12b1e8618397f57
Normal-4104.png,Normal,Normal/images/Normal-4104.png,Normal/masks/Normal-4104.png,True,G13058,train,df54f85e9227127963597d971795f159,d42d43531b0f4f2e
Normal-4105.png,Normal,Normal/images/Normal-4105.png,Normal/masks/Normal-4105.png,True,G13059,val,fa91dd5e57ccacbde16b8000671a2de7,d83d42433f0aec3d
Normal-4106.png,Normal,Normal/images/Normal-4106.png,Normal/masks/Normal-4106.png,True,G13060,test,b575a1dd58dd46411553105ba81f913b,897f565a421a2b3b
Normal-4107.png,Normal,Normal/images/Normal-4107.png,Normal/masks/Normal-4107.png,True,G13061,train,801e66f8e0dffc1a951977ffd02349b3,d93d86461b454f1b
Normal-4108.png,Normal,Normal/images/Normal-4108.png,Normal/masks/Normal-4108.png,True,G13062,train,7404da027186e8ba8ffab1022e40001d,dd2dc61a0c582fe5
Normal-4109.png,Normal,Normal/images/Normal-4109.png,Normal/masks/Normal-4109.png,True,G13063,val,d6be9c79d88737fb457fa5ec44a376c2,997947461a4f6e38
Normal-411.png,Normal,Normal/images/Normal-411.png,Normal/masks/Normal-411.png,True,G13064,test,3400aa055f1a0db1acb62e08751ad51d,913bc65ab85919d9
Normal-4110.png,Normal,Normal/images/Normal-4110.png,Normal/masks/Normal-4110.png,True,G13065,val,4e4be4b97584bee0d128d4afc1ee88d2,992c071bc07fb8e9
Normal-4111.png,Normal,Normal/images/Normal-4111.png,Normal/masks/Normal-4111.png,True,G13066,train,dc8eadcfcb42cb5712d58e393071e011,806807171a5f7f5b
Normal-4112.png,Normal,Normal/images/Normal-4112.png,Normal/masks/Normal-4112.png,True,G13067,train,825fccf2b426d70e3dd9fae062a8e8b1,d42fc30d4a0f0b6f
Normal-4113.png,Normal,Normal/images/Normal-4113.png,Normal/masks/Normal-4113.png,True,G13068,val,15b8db25db6e9574d695bacacff45693,d42f4a43cb09fda2
Normal-4114.png,Normal,Normal/images/Normal-4114.png,Normal/masks/Normal-4114.png,True,G13069,train,fdd1d3e68b5ac8e3c17c71104da2fb6d,dd78c6431e4e5b18
Normal-4115.png,Normal,Normal/images/Normal-4115.png,Normal/masks/Normal-4115.png,True,G13070,train,d56c9abf20fd12e1f0f139e35af8ad43,d979061a3979392d
Normal-4116.png,Normal,Normal/images/Normal-4116.png,Normal/masks/Normal-4116.png,True,G13071,train,bacaf983c789bfef501e759ef20f5458,d97b47905a1d5a19
Normal-4117.png,Normal,Normal/images/Normal-4117.png,Normal/masks/Normal-4117.png,True,G13072,train,0047d03fdb42a5061ab478cf8ba927b9,9968c6191f6a39ec
Normal-4118.png,Normal,Normal/images/Normal-4118.png,Normal/masks/Normal-4118.png,True,G13073,train,da9286c2e21a61665db51a142c87d7a1,91694e97593fa438
Normal-4119.png,Normal,Normal/images/Normal-4119.png,Normal/masks/Normal-4119.png,True,G13074,train,ec488dd8c7e4710dba9e1de368c36ca8,d969c61ed45a385a
Normal-412.png,Normal,Normal/images/Normal-412.png,Normal/masks/Normal-412.png,True,G13075,train,6a17839c5ff39dd263b1f5808ac6c82a,90abc31ff4600fe3
Normal-4120.png,Normal,Normal/images/Normal-4120.png,Normal/masks/Normal-4120.png,True,G13076,train,753db8d9c275a7791e2f51c9b0b82c51,897b061a5f49f8b1
Normal-4121.png,Normal,Normal/images/Normal-4121.png,Normal/masks/Normal-4121.png,True,G13077,train,7d8c262d133b4d32ab8d5228cd411cc7,dd2cd20b164b1d6e
Normal-4122.png,Normal,Normal/images/Normal-4122.png,Normal/masks/Normal-4122.png,True,G13078,val,39d7513e865565f41142ccbc110489d2,dd2ec2570a846f1d
Normal-4123.png,Normal,Normal/images/Normal-4123.png,Normal/masks/Normal-4123.png,True,G13079,test,0fd97dd3ba2fe629d9a254fc70a36588,915a8e5a0b757ec2
Normal-4124.png,Normal,Normal/images/Normal-4124.png,Normal/masks/Normal-4124.png,True,G13080,val,5ae791acd62c25bd4489573db291c1fd,ce35d34b53415b43
Normal-4125.png,Normal,Normal/images/Normal-4125.png,Normal/masks/Normal-4125.png,True,G13081,train,c9f5557314479ab10965b6bde68d51d3,887d035b386cfcb8
Normal-4126.png,Normal,Normal/images/Normal-4126.png,Normal/masks/Normal-4126.png,True,G13082,train,cc7eb40d835b14fc5c14378e46791ee8,8169d647190fce1f
Normal-4127.png,Normal,Normal/images/Normal-4127.png,Normal/masks/Normal-4127.png,True,G13083,train,df4781d2c65dd26d5f863b6e4aba1f81,dc3f421c5a2d7869
Normal-4128.png,Normal,Normal/images/Normal-4128.png,Normal/masks/Normal-4128.png,True,G13084,test,9701f109c9ff193b45e596f6187296ba,987d86161f477e18
Normal-4129.png,Normal,Normal/images/Normal-4129.png,Normal/masks/Normal-4129.png,True,G13085,test,961fa01ef32039690b092a256e19ee67,992f861b051f1aed
Normal-413.png,Normal,Normal/images/Normal-413.png,Normal/masks/Normal-413.png,True,G13086,train,67dc96283444d092c6d75c20590ffbd9,9d6f871b890dc8c5
Normal-4130.png,Normal,Normal/images/Normal-4130.png,Normal/masks/Normal-4130.png,True,G13087,val,19056ba8f214cddc1b57d59eebd2b693,997e071a946eb8e8
Normal-4131.png,Normal,Normal/images/Normal-4131.png,Normal/masks/Normal-4131.png,True,G13088,val,2475d6545c3f2f0ccd629e4f28495eb8,942e46c21be8cfec
Normal-4132.png,Normal,Normal/images/Normal-4132.png,Normal/masks/Normal-4132.png,True,G13089,test,8a593644b4f96871220397347bcc4df9,dd2e42530f4b1f0e
Normal-4133.png,Normal,Normal/images/Normal-4133.png,Normal/masks/Normal-4133.png,True,G13090,train,ba626b25507c1095e69998be79530c35,b87dc65a59455859
Normal-4134.png,Normal,Normal/images/Normal-4134.png,Normal/masks/Normal-4134.png,True,G13091,train,583f12890980df26553d1e4be349e0b8,897b071a1b297e39
Normal-4135.png,Normal,Normal/images/Normal-4135.png,Normal/masks/Normal-4135.png,True,G13092,train,18d931238d2f623859146b2bd64f299e,9969871e4a8e5b4d
Normal-4136.png,Normal,Normal/images/Normal-4136.png,Normal/masks/Normal-4136.png,True,G13093,train,32936844196d91d0389fd216f2c90704,fd0c460a1f2d7c39
Normal-4137.png,Normal,Normal/images/Normal-4137.png,Normal/masks/Normal-4137.png,True,G13094,train,2b866a1c24e8a6ae429c8c475496b961,c47c03511f7ea578
Normal-4138.png,Normal,Normal/images/Normal-4138.png,Normal/masks/Normal-4138.png,True,G13095,train,9d660327d2ca7464767513b743ef2933,91395e16591fec29
Normal-4139.png,Normal,Normal/images/Normal-4139.png,Normal/masks/Normal-4139.png,True,G13096,train,eb5c49015ede1de43515133f65aaeebe,99298e1e1f4e78b1
Normal-414.png,Normal,Normal/images/Normal-414.png,Normal/masks/Normal-414.png,True,G13097,train,bfd6778b193531f254f24c9123d9c68a,d43dc71f0c4c6ce4
Normal-4140.png,Normal,Normal/images/Normal-4140.png,Normal/masks/Normal-4140.png,True,G13098,train,98681faf2fe6ccdfbc4a08d98e16521d,d93b861e1f054b1d
Normal-4141.png,Normal,Normal/images/Normal-4141.png,Normal/masks/Normal-4141.png,True,G13099,test,a04499c5fbd0e9a701bc8858ab6aad67,d42f02435f0f6c6e
Normal-4142.png,Normal,Normal/images/Normal-4142.png,Normal/masks/Normal-4142.png,True,G13100,test,1a543d65dd70aa285f3cce1e4a2adf3b,d929d71a1a4e3aa9
Normal-4143.png,Normal,Normal/images/Normal-4143.png,Normal/masks/Normal-4143.png,True,G13101,test,6b573c85c47f6e5db2be0776052b8947,99af061f1618575d
Normal-4144.png,Normal,Normal/images/Normal-4144.png,Normal/masks/Normal-4144.png,True,G13102,train,58907c1ad8eb1012406701880405bc40,dd2f465f084d0f58
Normal-4145.png,Normal,Normal/images/Normal-4145.png,Normal/masks/Normal-4145.png,True,G13103,train,2dc6bdfc34802134d21c2f13f61fbf83,dd6c03c31b1d4f45
Normal-4146.png,Normal,Normal/images/Normal-4146.png,Normal/masks/Normal-4146.png,True,G13104,train,7046a43d1e37bf8847e1355218b3d7cc,993907869a1fc57a
Normal-4147.png,Normal,Normal/images/Normal-4147.png,Normal/masks/Normal-4147.png,True,G13105,train,647f681294fece4fb1a6f1a0cd6eab56,9979d65e11474f11
Normal-4148.png,Normal,Normal/images/Normal-4148.png,Normal/masks/Normal-4148.png,True,G13106,val,bddcf5740fa3a91c1783d9db37a26105,986c535b453e2d66
Normal-4149.png,Normal,Normal/images/Normal-4149.png,Normal/masks/Normal-4149.png,True,G13107,val,a625d8082df2352f77062133f9c58859,d02d421f0e2dbd9d
Normal-415.png,Normal,Normal/images/Normal-415.png,Normal/masks/Normal-415.png,True,G13108,val,d44843886e45cfaa2cd423b2af102058,d46e8f0d8d4f03c3
Normal-4150.png,Normal,Normal/images/Normal-4150.png,Normal/masks/Normal-4150.png,True,G13109,train,d22067e88cc137a8f5b2f9a37398b632,d43d43431c0b6f3d
Normal-4151.png,Normal,Normal/images/Normal-4151.png,Normal/masks/Normal-4151.png,True,G13110,train,fb4b685a2b57fa400cff3ad266c81e1c,997d461b5b464b19
Normal-4152.png,Normal,Normal/images/Normal-4152.png,Normal/masks/Normal-4152.png,True,G13111,train,dd59fcafc0f531d20d9851bdcbb82b4b,813856171f4e3eda
Normal-4153.png,Normal,Normal/images/Normal-4153.png,Normal/masks/Normal-4153.png,True,G13112,train,541313e0bad18cf49aa3ea0fe55065b5,c97b07861e95471e
Normal-4154.png,Normal,Normal/images/Normal-4154.png,Normal/masks/Normal-4154.png,True,G13113,train,624ce9e08473dc3452f6e39d20e13a6d,91688e17171e7a9e
Normal-4155.png,Normal,Normal/images/Normal-4155.png,Normal/masks/Normal-4155.png,True,G13114,train,50d257fc68a5fcf8724b9b1e051cea67,8978c658554d7957
Normal-4156.png,Normal,Normal/images/Normal-4156.png,Normal/masks/Normal-4156.png,True,G13115,train,6b73aafb3dae14a6ad9053c0954bdc6f,dd60421e465b656f
Normal-4157.png,Normal,Normal/images/Normal-4157.png,Normal/masks/Normal-4157.png,True,G13116,train,852b6c8ea12eee77d6d1feedd802aa70,9d78d31b4c0e390f
Normal-4158.png,Normal,Normal/images/Normal-4158.png,Normal/masks/Normal-4158.png,True,G13117,train,b4e01cde2707e1beaba54302f78abbf1,d43d430b4b2d5e2d
Normal-4159.png,Normal,Normal/images/Normal-4159.png,Normal/masks/Normal-4159.png,True,G13118,train,c83626589b8b156da61129faded27210,d939071a1e457b78
Normal-416.png,Normal,Normal/images/Normal-416.png,Normal/masks/Normal-416.png,True,G13119,train,3d82c6b7cb66ca01ac5413d44b380808,817d877a29693965
Normal-4160.png,Normal,Normal/images/Normal-4160.png,Normal/masks/Normal-4160.png,True,G13120,test,a403120ebddd9d8a343ba6e6881640c1,ce294b0f0f2e0ead
Normal-4161.png,Normal,Normal/images/Normal-4161.png,Normal/masks/Normal-4161.png,True,G13121,train,be6ea7999bb652aa675bc9a562509880,cc6ad34b4d4b6d48
Normal-4162.png,Normal,Normal/images/Normal-4162.png,Normal/masks/Normal-4162.png,True,G13122,test,9d42426bc4a0abb00e91748861251b9f,9979851e8d5e184f
Normal-4163.png,Normal,Normal/images/Normal-4163.png,Normal/masks/Normal-4163.png,True,G13123,val,b98878eff8ed0f67f8ba41520714f8e8,d16b865e5a1a5e1a
Normal-4164.png,Normal,Normal/images/Normal-4164.png,Normal/masks/Normal-4164.png,True,G13124,train,f584a9adbb6966f54e471895077c46a9,c869c65e5b453b19
Normal-4165.png,Normal,Normal/images/Normal-4165.png,Normal/masks/Normal-4165.png,True,G13125,train,9c44b0bdce674bf72a0d447b87dc3b4e,ec7cd35b54c21d88
Normal-4166.png,Normal,Normal/images/Normal-4166.png,Normal/masks/Normal-4166.png,True,G13126,train,808db68115f34aeb68747f8ad89f9cbb,dc2e534f424f4c0f
Normal-4167.png,Normal,Normal/images/Normal-4167.png,Normal/masks/Normal-4167.png,True,G13127,train,13f0375066efb50f0e03ae74fe662406,817f071e1d061f1f
Normal-4168.png,Normal,Normal/images/Normal-4168.png,Normal/masks/Normal-4168.png,True,G13128,train,484f19ab08c3553952ae715fd813a853,cc3d420f0ace2fbc
Normal-4169.png,Normal,Normal/images/Normal-4169.png,Normal/masks/Normal-4169.png,True,G13129,val,6339a3e3250e1fcad276f076e2d607f8,9969861a1d5f5a59
Normal-417.png,Normal,Normal/images/Normal-417.png,Normal/masks/Normal-417.png,True,G13130,test,e9a13095032cc03c9443a7af1cc0a288,d43dcf1dcd0c1cc8
Normal-4170.png,Normal,Normal/images/Normal-4170.png,Normal/masks/Normal-4170.png,True,G13131,train,934710454e81df0025bceea137a08d1b,9169f3c6561e96c4
Normal-4171.png,Normal,Normal/images/Normal-4171.png,Normal/masks/Normal-4171.png,True,G13132,train,2cf9a01c630bf4b43bef8e76434cc015,997d065a114f591f
Normal-4172.png,Normal,Normal/images/Normal-4172.png,Normal/masks/Normal-4172.png,True,G13133,train,ea7bdce07a8c6d82dd8304fea5c6c58a,9178070f187e4d7d
Normal-4173.png,Normal,Normal/images/Normal-4173.png,Normal/masks/Normal-4173.png,True,G13134,train,932e93f6453bf2b1f91216a09e8e6839,c97dc75a14491d69
Normal-4174.png,Normal,Normal/images/Normal-4174.png,Normal/masks/Normal-4174.png,True,G13135,train,2bb4d327118bf01531859a1785954bbc,9d3dc61b1c461b5c
Normal-4175.png,Normal,Normal/images/Normal-4175.png,Normal/masks/Normal-4175.png,True,G13136,train,1eac20007517e7789f10a1e3c9e6a2b1,997d461b554a1d4b
Normal-4176.png,Normal,Normal/images/Normal-4176.png,Normal/masks/Normal-4176.png,True,G13137,train,b48186ea1e514cbc05dcac6c05ffe648,d96dc41a451a9e4f
Normal-4177.png,Normal,Normal/images/Normal-4177.png,Normal/masks/Normal-4177.png,True,G13138,train,75f0d4528279677158b855fa51136848,997807965ac8fc78
Normal-4178.png,Normal,Normal/images/Normal-4178.png,Normal/masks/Normal-4178.png,True,G13139,train,716c64119098004a88190af2fde4e0ab,d42ec30b8caaacaf
Normal-4179.png,Normal,Normal/images/Normal-4179.png,Normal/masks/Normal-4179.png,True,G13140,train,a2683b7c72d144ce1c1d976ded87bd8c,d91915c31b1ee969
Normal-418.png,Normal,Normal/images/Normal-418.png,Normal/masks/Normal-418.png,True,G13141,train,ba42b4935f1b393b90f30088a4be6599,902bcf1bdd491cc9
Normal-4180.png,Normal,Normal/images/Normal-4180.png,Normal/masks/Normal-4180.png,True,G13142,val,e64a45a77f487a8c52ca070ab4163333,9939061a1bc57bb9
Normal-4181.png,Normal,Normal/images/Normal-4181.png,Normal/masks/Normal-4181.png,True,G13143,val,908412decb298362f847360a998a984d,916b861a161e1f5f
Normal-4182.png,Normal,Normal/images/Normal-4182.png,Normal/masks/Normal-4182.png,True,G13144,val,95fb7b2ef131904a5b834604a4586b38,817a17065a5b697b
Normal-4183.png,Normal,Normal/images/Normal-4183.png,Normal/masks/Normal-4183.png,True,G13145,test,d4c6f8c66fb53ef495f4532d48c4fd3d,987fc61a58511f56
Normal-4184.png,Normal,Normal/images/Normal-4184.png,Normal/masks/Normal-4184.png,True,G13146,test,ae7d659bb45cd47e2a6cf230d8f0f88f,817917865a1d5f4b
Normal-4185.png,Normal,Normal/images/Normal-4185.png,Normal/masks/Normal-4185.png,True,G13147,train,b05c91ba41ebaf25b06e4ceb0ae61394,817987161a5d7a7a
Normal-4186.png,Normal,Normal/images/Normal-4186.png,Normal/masks/Normal-4186.png,True,G13148,train,6782cb1407f6524f1f87ccd1340ed799,9168041e595f797b
Normal-4187.png,Normal,Normal/images/Normal-4187.png,Normal/masks/Normal-4187.png,True,G13149,train,c093bf40e042a196b6ec2569495fa173,91685e1f3869e9b8
Normal-4188.png,Normal,Normal/images/Normal-4188.png,Normal/masks/Normal-4188.png,True,G13150,train,947ba757195e70e670d1591c22554f52,d87b851e1a845f3a
Normal-4189.png,Normal,Normal/images/Normal-4189.png,Normal/masks/Normal-4189.png,True,G13151,train,44eb4518f111f6ee3597417f857ea8e2,d83d461f1c455b39
Normal-419.png,Normal,Normal/images/Normal-419.png,Normal/masks/Normal-419.png,True,G13152,train,4095032238407b127a2b257383e98209,987d873a871f0c0f
Normal-4190.png,Normal,Normal/images/Normal-4190.png,Normal/masks/Normal-4190.png,True,G13153,train,40dcd047644e2a5ed7a15275e3f83946,993d8619596e38f1
Normal-4191.png,Normal,Normal/images/Normal-4191.png,Normal/masks/Normal-4191.png,True,G13154,train,d6fe11440cf36cbf474158923a35cd4d,997843535bc74f90
Normal-4192.png,Normal,Normal/images/Normal-4192.png,Normal/masks/Normal-4192.png,True,G13155,train,818635f836c3f9e9953eb2d11b72e80f,d178c651194f2f39
Normal-4193.png,Normal,Normal/images/Normal-4193.png,Normal/masks/Normal-4193.png,True,G13156,train,54a62254eddb44ee2b2d6bf520894a69,d52a4ac3435f1d4d
Normal-4194.png,Normal,Normal/images/Normal-4194.png,Normal/masks/Normal-4194.png,True,G13157,train,5bc8fff9f4196f9dddfd4cb02532e80b,dc2dc2439f4aa99c
Normal-4195.png,Normal,Normal/images/Normal-4195.png,Normal/masks/Normal-4195.png,True,G13158,train,df399d4268a419ca2d830221c30495ac,d43d42435fd3434b
Normal-4196.png,Normal,Normal/images/Normal-4196.png,Normal/masks/Normal-4196.png,True,G13159,train,f35a6d205759c9150ea35ea8ae2d6a1e,817816fa79a1e9b1
Normal-4197.png,Normal,Normal/images/Normal-4197.png,Normal/masks/Normal-4197.png,True,G13160,train,33cb88e61bbf5f9b77617865a8004d8a,9905061e1f5e5f78
Normal-4198.png,Normal,Normal/images/Normal-4198.png,Normal/masks/Normal-4198.png,True,G13161,test,dabd3d9a9c7c0724145af2c9a78e3238,897887861a15cf5f
Normal-4199.png,Normal,Normal/images/Normal-4199.png,Normal/masks/Normal-4199.png,True,G13162,test,bb1d7b1da3aae28f51e0fe4abb4057e1,d82d42531b4f4f39
Normal-42.png,Normal,Normal/images/Normal-42.png,Normal/masks/Normal-42.png,True,G13163,val,223751f21ea859ec382766a977d1c0c9,9879861e191b595f
Normal-420.png,Normal,Normal/images/Normal-420.png,Normal/masks/Normal-420.png,True,G13164,val,26395a1da7b997ef6480060a02f9d1b1,9075a73aad396865
Normal-4200.png,Normal,Normal/images/Normal-4200.png,Normal/masks/Normal-4200.png,True,G13165,val,725f9feb96b936aa7d4386afa7d68876,c03f074b1f4b1a5e
Normal-4201.png,Normal,Normal/images/Normal-4201.png,Normal/masks/Normal-4201.png,True,G13166,train,3324cdfabb54f090b3af3948e2e8a7e0,d42e8115174e2f6f
Normal-4202.png,Normal,Normal/images/Normal-4202.png,Normal/masks/Normal-4202.png,True,G13167,train,de46eba5c621c5baf52acddb0d0d501f,912b84869ef8e9e9
Normal-4203.png,Normal,Normal/images/Normal-4203.png,Normal/masks/Normal-4203.png,True,G13168,test,f2fbfb33d7908d9d1d31f0c3f8bb0e92,992d861acd3ae06f
Normal-4204.png,Normal,Normal/images/Normal-4204.png,Normal/masks/Normal-4204.png,True,G13169,train,d58036eeb5491ed90d642fa7961b0f4c,997d861a095d78e9
Normal-4205.png,Normal,Normal/images/Normal-4205.png,Normal/masks/Normal-4205.png,True,G13170,test,1f6e3f9abcb5f3e077684ca5dee53a5f,91b907c73a957c85
Normal-4206.png,Normal,Normal/images/Normal-4206.png,Normal/masks/Normal-4206.png,True,G13171,train,184efd7dfe5bda2240959c280fc841b0,94b36b9c5c6994c3
Normal-4207.png,Normal,Normal/images/Normal-4207.png,Normal/masks/Normal-4207.png,True,G13172,train,1870076782c1f8dce84f9697f9b362c4,817b061e5a6a79ba
Normal-4208.png,Normal,Normal/images/Normal-4208.png,Normal/masks/Normal-4208.png,True,G13173,test,3c4bc98c4cd029d841ceb9c2bf34502f,803fd60f1a155f1d
Normal-4209.png,Normal,Normal/images/Normal-4209.png,Normal/masks/Normal-4209.png,True,G13174,train,076990e1437c62160bed71343f27ab83,997f861a151e191f
Normal-421.png,Normal,Normal/images/Normal-421.png,Normal/masks/Normal-421.png,True,G13175,test,ee0257c0c79fc29c9736c1f6ede23199,8166c67ae16d2dc5
Normal-4210.png,Normal,Normal/images/Normal-4210.png,Normal/masks/Normal-4210.png,True,G13176,train,1a98933590f9e58bc4f7a056cbe51e1e,9d78431b4d1eb84d
Normal-4211.png,Normal,Normal/images/Normal-4211.png,Normal/masks/Normal-4211.png,True,G13177,train,8d8984ef6719bc8a9d644d8fed07e97f,dd2ad2575b420f58
Normal-4212.png,Normal,Normal/images/Normal-4212.png,Normal/masks/Normal-4212.png,True,G13178,train,a4abc454b05e3aac4c55611ac2c562f9,993dc54217435e1f
Normal-4213.png,Normal,Normal/images/Normal-4213.png,Normal/masks/Normal-4213.png,True,G13179,train,ddf7d602f4add3fa129156488b9acef0,9a61659c67969939
Normal-4214.png,Normal,Normal/images/Normal-4214.png,Normal/masks/Normal-4214.png,True,G13180,train,5aacdab36963b9dd8d5cda4671c622da,8168d657de165547
Normal-4215.png,Normal,Normal/images/Normal-4215.png,Normal/masks/Normal-4215.png,True,G13181,train,2c8ea0cfbb999613d7e197f80fc02bda,9939d6571ac65e18
Normal-4216.png,Normal,Normal/images/Normal-4216.png,Normal/masks/Normal-4216.png,True,G13182,train,077792e00ec258eb9ae73410130225ca,913946171a5f7a1b
Normal-4217.png,Normal,Normal/images/Normal-4217.png,Normal/masks/Normal-4217.png,True,G13183,train,12de22d930bc740157563ac867854fb9,997f050716171f1e
Normal-4218.png,Normal,Normal/images/Normal-4218.png,Normal/masks/Normal-4218.png,True,G13184,val,d41c19f802afe0717f434c85316c0d43,c03dc3475c1ced39
Normal-4219.png,Normal,Normal/images/Normal-4219.png,Normal/masks/Normal-4219.png,True,G13185,val,121e3860d3552066fe0d982c4246afb7,d53d06455b4e6879
Normal-422.png,Normal,Normal/images/Normal-422.png,Normal/masks/Normal-422.png,True,G13186,val,9f0465d2ea0494f08695b7f11d0de47a,947ec75bbb852860
Normal-4220.png,Normal,Normal/images/Normal-4220.png,Normal/masks/Normal-4220.png,True,G13187,train,04ce62d2dfd62f0212c5db6e101b7a6d,807f071a1b2d7fa8
Normal-4221.png,Normal,Normal/images/Normal-4221.png,Normal/masks/Normal-4221.png,True,G13188,train,ac56bc848bfa94882749b03bd1883678,d16bc6d65a445a59
Normal-4222.png,Normal,Normal/images/Normal-4222.png,Normal/masks/Normal-4222.png,True,G13189,train,5a4aa8e66283980a23292ea2fa95cb2e,d928061b5bc77ad1
Normal-4223.png,Normal,Normal/images/Normal-4223.png,Normal/masks/Normal-4223.png,True,G13190,train,6f0c1fbc431a1c6c42312ee7ecfbcb7d,8178561c5b6fe985
Normal-4224.png,Normal,Normal/images/Normal-4224.png,Normal/masks/Normal-4224.png,True,G13191,train,13b8c22d90c0cb82e86c9ac80d6488d8,817b041e59c97b79
Normal-4225.png,Normal,Normal/images/Normal-4225.png,Normal/masks/Normal-4225.png,True,G13192,train,d010c87a739b9a7bd57b2d8ce0f35f52,bb2984125b5f555c
Normal-4226.png,Normal,Normal/images/Normal-4226.png,Normal/masks/Normal-4226.png,True,G13193,train,f3c455254decf6e7d89fd89332c005e8,d9bd461bd247124b
Normal-4227.png,Normal,Normal/images/Normal-4227.png,Normal/masks/Normal-4227.png,True,G13194,train,167c6e3028639a402c800951fc004da0,d52c479353435f1a
Normal-4228.png,Normal,Normal/images/Normal-4228.png,Normal/masks/Normal-4228.png,True,G13195,train,5e0f9faf1258bb6ef2805a9fce14400a,8939c71e1a557a39
Normal-4229.png,Normal,Normal/images/Normal-4229.png,Normal/masks/Normal-4229.png,True,G13196,train,ca9b8cf70603d5dea83ac6515f75bb6d,887d85121f4d1f1f
Normal-423.png,Normal,Normal/images/Normal-423.png,Normal/masks/Normal-423.png,True,G13197,train,5babc9550e33156a4df6015ff51abd95,917f861a095d58dd
Normal-4230.png,Normal,Normal/images/Normal-4230.png,Normal/masks/Normal-4230.png,True,G13198,val,783eff3c254dc8644baef0b71234a6d9,d93dc61b1e451a1d
Normal-4231.png,Normal,Normal/images/Normal-4231.png,Normal/masks/Normal-4231.png,True,G13199,train,0c003356960d2c4b08774d410a33b562,f93905561e1d4a5b
Normal-4232.png,Normal,Normal/images/Normal-4232.png,Normal/masks/Normal-4232.png,True,G13200,train,1db66f6e9412500453c0ce6a38715a29,9978878610875f5f
Normal-4233.png,Normal,Normal/images/Normal-4233.png,Normal/masks/Normal-4233.png,True,G13201,val,11ffe6d23e4bd178422216f73bb1fe48,d96f421e0d19a9ab
Normal-4234.png,Normal,Normal/images/Normal-4234.png,Normal/masks/Normal-4234.png,True,G13202,val,c23730dd927d5b2a8108033d543f0dbc,81aa07531d4f4f5d
Normal-4235.png,Normal,Normal/images/Normal-4235.png,Normal/masks/Normal-4235.png,True,G13203,train,8146f68bfc32ed10c04c55898ecf5bf8,dc7d420b0d4e2d3d
Normal-4236.png,Normal,Normal/images/Normal-4236.png,Normal/masks/Normal-4236.png,True,G13204,test,21d7e1c72d71f4e03b7539f17b409db0,9939c65e184d3b39
Normal-4237.png,Normal,Normal/images/Normal-4237.png,Normal/masks/Normal-4237.png,True,G13205,test,716b6d610e51372fbaed5fa31fe98055,cc3c425dc22f38af
Normal-4238.png,Normal,Normal/images/Normal-4238.png,Normal/masks/Normal-4238.png,True,G13206,train,a978887ed6cc968a17e67be59797f265,dc2f530b2a2f3865
Normal-4239.png,Normal,Normal/images/Normal-4239.png,Normal/masks/Normal-4239.png,True,G13207,test,d936fd9bfc10dadad588810dfe7d9670,cc3ec1424f4a3f2d
Normal-424.png,Normal,Normal/images/Normal-424.png,Normal/masks/Normal-424.png,True,G13208,test,ee21b25f14bf14c7ec0901f06621c932,d12ace1a5d791d1a
Normal-4240.png,Normal,Normal/images/Normal-4240.png,Normal/masks/Normal-4240.png,True,G13209,train,4fbb9eed3e678c092420c8e99a18fc53,d978d21b19c60f39
Normal-4241.png,Normal,Normal/images/Normal-4241.png,Normal/masks/Normal-4241.png,True,G13210,test,f2f510b2ef0105cb990a37208e37871c,992fc516944f1e4b
Normal-4242.png,Normal,Normal/images/Normal-4242.png,Normal/masks/Normal-4242.png,True,G13211,train,cf4c5e1550557609c0fa6a643a5402f2,916987161f4f1e9a
Normal-4243.png,Normal,Normal/images/Normal-4243.png,Normal/masks/Normal-4243.png,True,G13212,train,fc34f0adb59f4dd5a990153e9f604ba6,dd6c435b5f441a1c
Normal-4244.png,Normal,Normal/images/Normal-4244.png,Normal/masks/Normal-4244.png,True,G13213,train,8c874ddcddb8c82dcb32344508323955,817a16081bc77f9d
Normal-4245.png,Normal,Normal/images/Normal-4245.png,Normal/masks/Normal-4245.png,True,G13214,train,f1d70e4992dfb832ddd958060fc253a1,ac7dc24b574a1f44
Normal-4246.png,Normal,Normal/images/Normal-4246.png,Normal/masks/Normal-4246.png,True,G13215,train,a4b76415cccd169f4721ea64be285e4e,986d061e471e5d5e
Normal-4247.png,Normal,Normal/images/Normal-4247.png,Normal/masks/Normal-4247.png,True,G13216,train,dabc4adef58e06bd1b12ad82d5c87c3e,9168c75a1f493d39
Normal-4248.png,Normal,Normal/images/Normal-4248.png,Normal/masks/Normal-4248.png,True,G13217,train,984183f679993d026ae70f9cc59e33b0,d93dc5431f0d4b0b
Normal-4249.png,Normal,Normal/images/Normal-4249.png,Normal/masks/Normal-4249.png,True,G13218,train,4caceacd77f6c58f03a4d2080115ae57,9178161e197d79b1
Normal-425.png,Normal,Normal/images/Normal-425.png,Normal/masks/Normal-425.png,True,G13219,train,abe9194b3df61921a5640b3709232018,d12ce23ae96d386c
Normal-4250.png,Normal,Normal/images/Normal-4250.png,Normal/masks/Normal-4250.png,True,G13220,train,c8590a325c41242c3e28e49c3ca4779c,d13d564a1919f92d
Normal-4251.png,Normal,Normal/images/Normal-4251.png,Normal/masks/Normal-4251.png,True,G13221,train,b436950be87edfe5408f0ae541927313,dd29861e5c4d5a59
Normal-4252.png,Normal,Normal/images/Normal-4252.png,Normal/masks/Normal-4252.png,True,G13222,train,8ba17e14ee11954dc68b6eefc959e584,d43c42d75b424f1b
Normal-4253.png,Normal,Normal/images/Normal-4253.png,Normal/masks/Normal-4253.png,True,G13223,train,df6e27a9e57e949596c5b911a628c36c,d93d46071b0e4f39
Normal-4254.png,Normal,Normal/images/Normal-4254.png,Normal/masks/Normal-4254.png,True,G13224,val,871a51eebfd4943cf4c1c719a7cc7c2b,985b041e5b7a78e9
Normal-4255.png,Normal,Normal/images/Normal-4255.png,Normal/masks/Normal-4255.png,True,G13225,val,7163e6ba4eba898af7d5be20188e1ccb,9978851716845fbd
Normal-4256.png,Normal,Normal/images/Normal-4256.png,Normal/masks/Normal-4256.png,True,G13226,val,65707bc270b077ec34236c59f9fdfc19,9d28d719c41f784e
Normal-4257.png,Normal,Normal/images/Normal-4257.png,Normal/masks/Normal-4257.png,True,G13227,test,82942779c8d824fff2a1e5250e992066,993b161a557878e5
Normal-4258.png,Normal,Normal/images/Normal-4258.png,Normal/masks/Normal-4258.png,True,G13228,test,eafb10295a28d5859bc3e52a20208ac1,dc2fc29bdc41385c
Normal-4259.png,Normal,Normal/images/Normal-4259.png,Normal/masks/Normal-4259.png,True,G13229,train,b5323edb3c30e60d038d48b898234911,9978961a1d7a39e4
Normal-426.png,Normal,Normal/images/Normal-426.png,Normal/masks/Normal-426.png,True,G13230,train,051ae77c8a6c164f75d0e43377d7e5e0,9029c61eda493df9
Normal-4260.png,Normal,Normal/images/Normal-4260.png,Normal/masks/Normal-4260.png,True,G13231,train,60fde917ce414305fd05a02d25088072,8178161b5b59393f
Normal-4261.png,Normal,Normal/images/Normal-4261.png,Normal/masks/Normal-4261.png,True,G13232,train,b713d21f28853d289c612baae16166ed,9568045f3a5e7a59
Normal-4262.png,Normal,Normal/images/Normal-4262.png,Normal/masks/Normal-4262.png,True,G13233,train,35c447894acc9a40045449af6a305819,dc2d425f0a5cad3a
Normal-4263.png,Normal,Normal/images/Normal-4263.png,Normal/masks/Normal-4263.png,True,G13234,train,5e4bb2f76263a677feaf7a66a61cbaf4,dd2cd0039b430bbf
Normal-4264.png,Normal,Normal/images/Normal-4264.png,Normal/masks/Normal-4264.png,True,G13235,train,03caf16277f80111381d799d725a1848,bd6cc21b454f184f
Normal-4265.png,Normal,Normal/images/Normal-4265.png,Normal/masks/Normal-4265.png,True,G13236,val,68e7f8d50017501aa0e4807c27c9d6c3,dc2e524b0a564f5d
Normal-4266.png,Normal,Normal/images/Normal-4266.png,Normal/masks/Normal-4266.png,True,G13237,train,8367f3d774971e71b24968b878f6a671,d87b45961e015b1f
Normal-4267.png,Normal,Normal/images/Normal-4267.png,Normal/masks/Normal-4267.png,True,G13238,train,f4a7f4c597d2ac346e5494c6c16d9f28,c97bd69618184b1f
Normal-4268.png,Normal,Normal/images/Normal-4268.png,Normal/masks/Normal-4268.png,True,G13239,test,01e32426fe1dba2de95e84c33b49bd2d,815b061a593b3baf
Normal-4269.png,Normal,Normal/images/Normal-4269.png,Normal/masks/Normal-4269.png,True,G13240,test,a97e8369a602ee124412b26fb87bb5f0,fd2cd29b0ac21b1b
Normal-427.png,Normal,Normal/images/Normal-427.png,Normal/masks/Normal-427.png,True,G13241,train,dfba752719a3c31e71e10ac280422cf1,816f8f0bc9c372d2
Normal-4270.png,Normal,Normal/images/Normal-4270.png,Normal/masks/Normal-4270.png,True,G13242,train,a3bee09d3ea3cae5c751070224d050a4,dd2e42431f0c6b3d
Normal-4271.png,Normal,Normal/images/Normal-4271.png,Normal/masks/Normal-4271.png,True,G13243,test,32b9ffc6cae938f678ad12c2009f47a0,cc2ec24f5b420f5d
Normal-4272.png,Normal,Normal/images/Normal-4272.png,Normal/masks/Normal-4272.png,True,G13244,test,1a934c0fc0d9d14619697f2827ae8f0e,dc3c430c0b2dbcaf
Normal-4273.png,Normal,Normal/images/Normal-4273.png,Normal/masks/Normal-4273.png,True,G13245,train,d09d02245f09c28b7e64bbcefa7da208,9969871a0d5e794d
Normal-4274.png,Normal,Normal/images/Normal-4274.png,Normal/masks/Normal-4274.png,True,G13246,train,62d1959c25ddb2d4453c6ffcb48ca30e,9d6c821bd1cf2d92
Normal-4275.png,Normal,Normal/images/Normal-4275.png,Normal/masks/Normal-4275.png,True,G13247,val,647138d38745ca83dfcef02a6e1be132,d97b961a995ab901
Normal-4276.png,Normal,Normal/images/Normal-4276.png,Normal/masks/Normal-4276.png,True,G13248,val,01f63949074fc17908379d96bd142d8a,91539656185f5f1a
Normal-4277.png,Normal,Normal/images/Normal-4277.png,Normal/masks/Normal-4277.png,True,G13249,train,5eaf021a8c9fc1d945deae35f36f299e,9979061a1d5f39b8
Normal-4278.png,Normal,Normal/images/Normal-4278.png,Normal/masks/Normal-4278.png,True,G13250,train,6bdb3abf330684d9b8f3b197253999a5,d92dc61b5d4d3891
Normal-4279.png,Normal,Normal/images/Normal-4279.png,Normal/masks/Normal-4279.png,True,G13251,train,7d1b86c88f299a149563f545b67645dc,817b06065a5de87f
Normal-428.png,Normal,Normal/images/Normal-428.png,Normal/masks/Normal-428.png,True,G13252,train,ccfc6794b16c636c522b14d74079040a,d03dc71dcccd18d2
Normal-4280.png,Normal,Normal/images/Normal-4280.png,Normal/masks/Normal-4280.png,True,G13253,train,03407c01a1119bb025df9075ee1c92ca,c93d460e1aec6ab9
Normal-4281.png,Normal,Normal/images/Normal-4281.png,Normal/masks/Normal-4281.png,True,G13254,test,40e082e5338c3356ff8ce7ed4080afbb,d12dc2171e474b1f
Normal-4282.png,Normal,Normal/images/Normal-4282.png,Normal/masks/Normal-4282.png,True,G13255,train,bc8d3aa9b3dd9b05cc0a965dbfa968c5,9976471b471b3c0b
Normal-4283.png,Normal,Normal/images/Normal-4283.png,Normal/masks/Normal-4283.png,True,G13256,train,68c1186b20f49741ba8346f1c058998c,997bc71a084d7a59
Normal-4284.png,Normal,Normal/images/Normal-4284.png,Normal/masks/Normal-4284.png,True,G13257,train,d4ad892a56d546a673234270470306b9,913b565e1a4d1f19
Normal-4285.png,Normal,Normal/images/Normal-4285.png,Normal/masks/Normal-4285.png,True,G13258,train,86fedc25868329781b88eaa5756847cb,dc2ec31f471f410d
Normal-4286.png,Normal,Normal/images/Normal-4286.png,Normal/masks/Normal-4286.png,True,G13259,val,36260a4af85bd2f6a62c28c146b955f1,dd2ec3130f454e5c
Normal-4287.png,Normal,Normal/images/Normal-4287.png,Normal/masks/Normal-4287.png,True,G13260,train,a48bca3c248db615e084c05f32a60821,d46e03534f4a0f6d
Normal-4288.png,Normal,Normal/images/Normal-4288.png,Normal/masks/Normal-4288.png,True,G13261,train,1d23620bf8a3211b7b42f472c6938185,997d471a55431f49
Normal-4289.png,Normal,Normal/images/Normal-4289.png,Normal/masks/Normal-4289.png,True,G13262,train,3e818138eb04a567510384fc25b0ceb2,d93dc61219451e9f
Normal-429.png,Normal,Normal/images/Normal-429.png,Normal/masks/Normal-429.png,True,G13263,val,b001dd0f8de94eb6e267b91bad08de77,d02fc65f5e49191c
Normal-4290.png,Normal,Normal/images/Normal-4290.png,Normal/masks/Normal-4290.png,True,G13264,test,17ec5796eb08b65bbf2d1742166d5f98,992cc65b1bc74e38
Normal-4291.png,Normal,Normal/images/Normal-4291.png,Normal/masks/Normal-4291.png,True,G13265,train,b5b8b2de64db9df27724094ba40cead0,c82f578f1b2da838
Normal-4292.png,Normal,Normal/images/Normal-4292.png,Normal/masks/Normal-4292.png,True,G13266,train,ad2312eee0dc7fe20789ee8f4487d54a,cc3d421b5e4e38ec
Normal-4293.png,Normal,Normal/images/Normal-4293.png,Normal/masks/Normal-4293.png,True,G13267,train,ac6f6b524389e9f1f4b065eae28d2b7b,883f521b564f1b4d
Normal-4294.png,Normal,Normal/images/Normal-4294.png,Normal/masks/Normal-4294.png,True,G13268,train,6bb9d7ce2e310282fa9154fe46d44f38,817916165b7de863
Normal-4295.png,Normal,Normal/images/Normal-4295.png,Normal/masks/Normal-4295.png,True,G13269,val,94c7c280800b62eb5a34a080810a3769,f91716180939abaf
Normal-4296.png,Normal,Normal/images/Normal-4296.png,Normal/masks/Normal-4296.png,True,G13270,train,bfdd9cd7417ce53f48567446a407cd7a,dd3cd30b0d4a1d0f
Normal-4297.png,Normal,Normal/images/Normal-4297.png,Normal/masks/Normal-4297.png,True,G13271,train,b3424703e57892a83ddf81d58b9e80aa,dd3ed20b0d0c3bac
Normal-4298.png,Normal,Normal/images/Normal-4298.png,Normal/masks/Normal-4298.png,True,G13272,train,6d7dea7b7fe3a2424055207e5a627298,d528c61b5e097f0d
Normal-4299.png,Normal,Normal/images/Normal-4299.png,Normal/masks/Normal-4299.png,True,G13273,train,a4a837af892fa306713c24201eef52f9,c87dc756964b1658
Normal-43.png,Normal,Normal/images/Normal-43.png,Normal/masks/Normal-43.png,True,G13274,train,655f5874e6d16c7e7715db1f0d29f9b4,997b071a59eda8d0
Normal-430.png,Normal,Normal/images/Normal-430.png,Normal/masks/Normal-430.png,True,G13275,train,8fd7f80199d875756ec34727254fecb4,812dc278f87979c9
Normal-4300.png,Normal,Normal/images/Normal-4300.png,Normal/masks/Normal-4300.png,True,G13276,train,afa4b68b380c1b62942ce170f9e12e9d,c86fc65a54473a55
Normal-4301.png,Normal,Normal/images/Normal-4301.png,Normal/masks/Normal-4301.png,True,G13277,train,68530a0f068a8bce0c4f80bd73d2c345,c12c061b1f5e5b5b
Normal-4302.png,Normal,Normal/images/Normal-4302.png,Normal/masks/Normal-4302.png,True,G13278,train,2406b541c0f84d2e3c3b3cbef6072337,993e961a591b5dc2
Normal-4303.png,Normal,Normal/images/Normal-4303.png,Normal/masks/Normal-4303.png,True,G13279,test,3cabede026acdf2d082679d52de4a148,d03d432e2e3536b3
Normal-4304.png,Normal,Normal/images/Normal-4304.png,Normal/masks/Normal-4304.png,True,G13280,val,f7823e64a6fa98bdc857bf3c0379f8c2,997b16181e5b195b
Normal-4305.png,Normal,Normal/images/Normal-4305.png,Normal/masks/Normal-4305.png,True,G13281,test,cf38a1d4bebd9f8810748973f72c3cc0,8029c71e174f5f17
Normal-4306.png,Normal,Normal/images/Normal-4306.png,Normal/masks/Normal-4306.png,True,G13282,train,debd33c02640ff95435540e1f9015168,ddf8088b3a8f3c91
Normal-4307.png,Normal,Normal/images/Normal-4307.png,Normal/masks/Normal-4307.png,True,G13283,train,b517321c836c1902364829936e530c9e,fd68c65e584a7a30
Normal-4308.png,Normal,Normal/images/Normal-4308.png,Normal/masks/Normal-4308.png,True,G13284,train,7bac88cb44cf694aaf65b7aaafac0eb6,c97d061e191d1d5d
Normal-4309.png,Normal,Normal/images/Normal-4309.png,Normal/masks/Normal-4309.png,True,G13285,test,582c8e3b01c9f9db71fbf3ecc66ad6ae,917b061a195d5b1f
Normal-431.png,Normal,Normal/images/Normal-431.png,Normal/masks/Normal-431.png,True,G13286,val,fe41bdc81d1694dc1d7cedb5f70b620e,9caf871e0b1d4c4d
Normal-4310.png,Normal,Normal/images/Normal-4310.png,Normal/masks/Normal-4310.png,True,G13287,val,7af483240a52712c23025db4dfd670af,9d69ee16914b861e
Normal-4311.png,Normal,Normal/images/Normal-4311.png,Normal/masks/Normal-4311.png,True,G13288,test,f064eff85efb9829e3333b73e464c1ed,807f961e1f1a1f1a
Normal-4312.png,Normal,Normal/images/Normal-4312.png,Normal/masks/Normal-4312.png,True,G13289,train,61f276f7b11324e467e7337033d5b8d1,dd38c35b454b194b
Normal-4313.png,Normal,Normal/images/Normal-4313.png,Normal/masks/Normal-4313.png,True,G13290,train,ad6aa1f5210c868ef06f3081ac60d038,842e0757195d6c7e
Normal-4314.png,Normal,Normal/images/Normal-4314.png,Normal/masks/Normal-4314.png,True,G13291,train,cf6e0acfe0a00e47a49a82d6128f3334,c82dc61a1e5a38fd
Normal-4315.png,Normal,Normal/images/Normal-4315.png,Normal/masks/Normal-4315.png,True,G13292,train,98947cac57071cd26ce13b725189bd95,c13dcf96185b4e38
Normal-4316.png,Normal,Normal/images/Normal-4316.png,Normal/masks/Normal-4316.png,True,G13293,train,87a679a63a527dbbe3dc0cfa3c5d1592,9169861e11567e7b
Normal-4317.png,Normal,Normal/images/Normal-4317.png,Normal/masks/Normal-4317.png,True,G13294,train,db3127c5b03d1653765e15d03c27c848,817d065a195f5e5a
Normal-4318.png,Normal,Normal/images/Normal-4318.png,Normal/masks/Normal-4318.png,True,G13295,train,eb1cfe3c1c74b5ad5108c55c00aad373,dd28471b5d4d1c1e
Normal-4319.png,Normal,Normal/images/Normal-4319.png,Normal/masks/Normal-4319.png,True,G13296,train,1f4adaaa567c30f0d5fa58c2e477f30c,997d865a1acd5a38
Normal-432.png,Normal,Normal/images/Normal-432.png,Normal/masks/Normal-432.png,True,G13297,val,5f913fbe7d299d568e6fea8750b97bd8,9129c61aba6b396d
Normal-4320.png,Normal,Normal/images/Normal-4320.png,Normal/masks/Normal-4320.png,True,G13298,train,a223cc1979f0a5936ce84001139d00ec,c83b561f504f1b59
Normal-4321.png,Normal,Normal/images/Normal-4321.png,Normal/masks/Normal-4321.png,True,G13299,train,7adf1342c71d8986e550609af2cd0a0e,d43d030b0e0f4f6f
Normal-4322.png,Normal,Normal/images/Normal-4322.png,Normal/masks/Normal-4322.png,True,G13300,train,4ae563f0ad9a9b9969a3ec2739c7e163,d43d42430b4a6f3f
Normal-4323.png,Normal,Normal/images/Normal-4323.png,Normal/masks/Normal-4323.png,True,G13301,train,6fa9800b94046cbe6c229d3c4416caeb,983dc41e495e1de9
Normal-4324.png,Normal,Normal/images/Normal-4324.png,Normal/masks/Normal-4324.png,True,G13302,val,577be21603378fd14af338a91f21043a,9979851e194d7a39
Normal-4325.png,Normal,Normal/images/Normal-4325.png,Normal/masks/Normal-4325.png,True,G13303,train,c89c1924d3b869805af7363c9bb8cde3,dd3dc21f544a3c4a
Normal-4326.png,Normal,Normal/images/Normal-4326.png,Normal/masks/Normal-4326.png,True,G13304,train,aaa11c5a9679f3aec1e6835baeb9573b,d03d4e8439b774d2
Normal-4327.png,Normal,Normal/images/Normal-4327.png,Normal/masks/Normal-4327.png,True,G13305,val,67a0e9090814c9d06ff65d5f6c12fa8d,916986965a59557d
Normal-4328.png,Normal,Normal/images/Normal-4328.png,Normal/masks/Normal-4328.png,True,G13306,test,046fabefc61379169c9467f350021de7,dc3dc34715c30f94
Normal-4329.png,Normal,Normal/images/Normal-4329.png,Normal/masks/Normal-4329.png,True,G13307,train,3e87fdfc61cf8576136e511b02d1ef8b,c92f861e1c4e5b39
Normal-433.png,Normal,Normal/images/Normal-433.png,Normal/masks/Normal-433.png,True,G13308,test,a7745bbff0d8820791851c0f52d3f413,9168c71e98cd79e1
Normal-4330.png,Normal,Normal/images/Normal-4330.png,Normal/masks/Normal-4330.png,True,G13309,train,5dfa58280a2f742b94bdd6e3d80c2764,dd3e571a544b1c4c
Normal-4331.png,Normal,Normal/images/Normal-4331.png,Normal/masks/Normal-4331.png,True,G13310,train,9359b7942252fab1f1fc640a2e1d22a8,dd3dd2431b0c4b1d
Normal-4332.png,Normal,Normal/images/Normal-4332.png,Normal/masks/Normal-4332.png,True,G13311,train,bb428d1f2c493730891733bab9230b32,817a56571b460f5b
Normal-4333.png,Normal,Normal/images/Normal-4333.png,Normal/masks/Normal-4333.png,True,G13312,train,a5c8fe5eeef37a2c5eac8719fe9e4db7,c4240f3f74613773
Normal-4334.png,Normal,Normal/images/Normal-4334.png,Normal/masks/Normal-4334.png,True,G13313,train,ab28646796a24787129885bd0ac3f077,dd2cd20b1c4b1bbc
Normal-4335.png,Normal,Normal/images/Normal-4335.png,Normal/masks/Normal-4335.png,True,G13314,val,36bd5f197f1c1b3ba66bd8992f405f5e,cd3dd25f024a1f92
Normal-4336.png,Normal,Normal/images/Normal-4336.png,Normal/masks/Normal-4336.png,True,G13315,train,148bf22f8c3161b7244f596dda98fe69,913dc612194f5f39
Normal-4337.png,Normal,Normal/images/Normal-4337.png,Normal/masks/Normal-4337.png,True,G13316,train,63ce6c4b0231ed2f7d9ee1c0e10001b4,8c45c30f2e3a3d75
Normal-4338.png,Normal,Normal/images/Normal-4338.png,Normal/masks/Normal-4338.png,True,G13317,train,baf1bf791fa666b25b8e147b5f65458a,c979045e1e4d5f38
Normal-4339.png,Normal,Normal/images/Normal-4339.png,Normal/masks/Normal-4339.png,True,G13318,train,02e1e85092592ca4c091ebf2ff6855e0,dd2ec64b1d4c4a1d
Normal-434.png,Normal,Normal/images/Normal-434.png,Normal/masks/Normal-434.png,True,G13319,train,989b3066f9d49a3faf27277fa587f0fe,91a9c61acd79594d
Normal-4340.png,Normal,Normal/images/Normal-4340.png,Normal/masks/Normal-4340.png,True,G13320,train,43833eb92661d720265edc68d6e479da,812b061e5b5d5e5d
Normal-4341.png,Normal,Normal/images/Normal-4341.png,Normal/masks/Normal-4341.png,True,G13321,test,699098ac82ad817bea0c6070635eda7b,9369c4961a5add5a
Normal-4342.png,Normal,Normal/images/Normal-4342.png,Normal/masks/Normal-4342.png,True,G13322,test,60c3c6974b85955a3ae61095127d064d,c0c95fde5a39c161
Normal-4343.png,Normal,Normal/images/Normal-4343.png,Normal/masks/Normal-4343.png,True,G13323,train,04a145fa7d394aa52fd4ee9888c5b335,807b46465a1def1b
Normal-4344.png,Normal,Normal/images/Normal-4344.png,Normal/masks/Normal-4344.png,True,G13324,train,c4a3646bf35abe01fc5de5d8d11f4e85,d92f461d1c4f1d49
Normal-4345.png,Normal,Normal/images/Normal-4345.png,Normal/masks/Normal-4345.png,True,G13325,test,76867cc38a6bf7fef6e6814b8fb12c83,d83bd71a1a59183d
Normal-4346.png,Normal,Normal/images/Normal-4346.png,Normal/masks/Normal-4346.png,True,G13326,train,a2a1c336191a6ce2878376a1e547c6ab,d43c02d74b0bcd4f
Normal-4347.png,Normal,Normal/images/Normal-4347.png,Normal/masks/Normal-4347.png,True,G13327,val,5f0b2d1b4a6e053282be8ef158beb3b9,cc3d434b1d4b4f0e
Normal-4348.png,Normal,Normal/images/Normal-4348.png,Normal/masks/Normal-4348.png,True,G13328,val,da278369b9d1380fb15f3b92ac54d6dc,893bc61e1c4f1b3c
Normal-4349.png,Normal,Normal/images/Normal-4349.png,Normal/masks/Normal-4349.png,True,G13329,train,aab843bc1ad645f5fa21e44a7665a311,95680e1f79a8ad2d
Normal-435.png,Normal,Normal/images/Normal-435.png,Normal/masks/Normal-435.png,True,G13330,train,d3dfd71e3dd8ae8ad29c9bb0afab4d3d,813bc65b9859399b
Normal-4350.png,Normal,Normal/images/Normal-4350.png,Normal/masks/Normal-4350.png,True,G13331,train,5436f0569bb98561b376b0893ab49954,817917561a5dea78
Normal-4351.png,Normal,Normal/images/Normal-4351.png,Normal/masks/Normal-4351.png,True,G13332,train,b736f06b082efd21df1d52b36a8ec6d1,d13fc6191aeb38c2
Normal-4352.png,Normal,Normal/images/Normal-4352.png,Normal/masks/Normal-4352.png,True,G13333,train,b2768ff38a36a248598d9c9c43916d9d,912c8617596f6b3a
Normal-4353.png,Normal,Normal/images/Normal-4353.png,Normal/masks/Normal-4353.png,True,G13334,train,99366c1aac0229e73e7056dfc469dfa8,9528561e5b5e784b
Normal-4354.png,Normal,Normal/images/Normal-4354.png,Normal/masks/Normal-4354.png,True,G13335,train,86baa1578983d56f9042fcda2d3b9e5f,dc2ad34b4f4c2e1c
Normal-4355.png,Normal,Normal/images/Normal-4355.png,Normal/masks/Normal-4355.png,True,G13336,train,d6b612a33b2122bdf99b650d277d3688,d979c6161a09ad3e
Normal-4356.png,Normal,Normal/images/Normal-4356.png,Normal/masks/Normal-4356.png,True,G13337,val,3c41f48ec8adcdf5a347d0eef11c1c3d,997dc45b13471b1c
Normal-4357.png,Normal,Normal/images/Normal-4357.png,Normal/masks/Normal-4357.png,True,G13338,train,0c523b5e6af64d83a70f78e90619dca7,817f07161e1f1f19
Normal-4358.png,Normal,Normal/images/Normal-4358.png,Normal/masks/Normal-4358.png,True,G13339,train,e234faa4dcea19e0dc2dc0df4d29db24,dc2dc2531dc15a1f
Normal-4359.png,Normal,Normal/images/Normal-4359.png,Normal/masks/Normal-4359.png,True,G13340,test,839b2e07b3ea11989606b271a9e3213b,992cc61b1f451abd
Normal-436.png,Normal,Normal/images/Normal-436.png,Normal/masks/Normal-436.png,True,G13341,test,9646353a842d3d4734031b60e9cabbc7,d03ce71bad4d3c48
Normal-4360.png,Normal,Normal/images/Normal-4360.png,Normal/masks/Normal-4360.png,True,G13342,train,64cba68704aa17fec89e0950c6ac53b6,cdbf8218c61bc0cf
Normal-4361.png,Normal,Normal/images/Normal-4361.png,Normal/masks/Normal-4361.png,True,G13343,test,329d82af6c15b6ae3810fba5def4a87f,d929460f1d1c6f39
Normal-4362.png,Normal,Normal/images/Normal-4362.png,Normal/masks/Normal-4362.png,True,G13344,val,19aa4facd2e26b9452e9820934731171,dd2cc34b4c2e3c2e
Normal-4363.png,Normal,Normal/images/Normal-4363.png,Normal/masks/Normal-4363.png,True,G13345,test,e28d484143e0316dc32753d4d1b505df,9979461f18a83bb9
Normal-4364.png,Normal,Normal/images/Normal-4364.png,Normal/masks/Normal-4364.png,True,G13346,train,437dd720d6d90bcfc8c3c716ae3a8742,dd3cc24b5c4a1a7c
Normal-4365.png,Normal,Normal/images/Normal-4365.png,Normal/masks/Normal-4365.png,True,G13347,val,113db1e5bb6d428c84f7b7de5efd57ba,d83f431f1a0d3f18
Normal-4366.png,Normal,Normal/images/Normal-4366.png,Normal/masks/Normal-4366.png,True,G13348,train,447de78e7eda46798b777c253262cf60,dd2ec21b51662a3d
Normal-4367.png,Normal,Normal/images/Normal-4367.png,Normal/masks/Normal-4367.png,True,G13349,train,b437fc1879f14ea3cc43626e03c8217b,c87dc69712450f1f
Normal-4368.png,Normal,Normal/images/Normal-4368.png,Normal/masks/Normal-4368.png,True,G13350,val,01b292469b56fcd1d886743bfdf0a7d8,9d78071a5b4d6e38
Normal-4369.png,Normal,Normal/images/Normal-4369.png,Normal/masks/Normal-4369.png,True,G13351,train,df64fae266fda76dc5d50166d57c9d42,9929861f1f075f19
Normal-437.png,Normal,Normal/images/Normal-437.png,Normal/masks/Normal-437.png,True,G13352,train,85fd6f31818609e654889b95969d6fbb,d46cc31fcf2c1cc4
Normal-4370.png,Normal,Normal/images/Normal-4370.png,Normal/masks/Normal-4370.png,True,G13353,train,2bfbbaef0961d044c245bb6326967051,817d0f1e585c5a79
Normal-4371.png,Normal,Normal/images/Normal-4371.png,Normal/masks/Normal-4371.png,True,G13354,train,cfbdace41c87215c56c54fc41e7407b7,dd38420f4a2f3c2f
Normal-4372.png,Normal,Normal/images/Normal-4372.png,Normal/masks/Normal-4372.png,True,G13355,train,52082bcba74f478416141ea9498eeedc,b96d861a495e584f
Normal-4373.png,Normal,Normal/images/Normal-4373.png,Normal/masks/Normal-4373.png,True,G13356,train,a4d3c5be73d775f508a7b7a5034458a6,c93d561e164b1e4e
Normal-4374.png,Normal,Normal/images/Normal-4374.png,Normal/masks/Normal-4374.png,True,G13357,test,3c3b3ec2e319bc04bb0c6eb961c14255,dd2ad3431f47451c
Normal-4375.png,Normal,Normal/images/Normal-4375.png,Normal/masks/Normal-4375.png,True,G13358,train,a7919654fd9d5e24681de4864dda1937,9178c6d35b044b7b
Normal-4376.png,Normal,Normal/images/Normal-4376.png,Normal/masks/Normal-4376.png,True,G13359,train,d784ff9826c55fa906c173474c27cb73,b979851e1a1d4b1d
Normal-4377.png,Normal,Normal/images/Normal-4377.png,Normal/masks/Normal-4377.png,True,G13360,train,b1640f71b525b96be86d7178a111f1f6,dc2cc24b0f4e1f3c
Normal-4378.png,Normal,Normal/images/Normal-4378.png,Normal/masks/Normal-4378.png,True,G13361,train,14dd3084a37694607e306eef4861af70,917a161a191b3bbe
Normal-4379.png,Normal,Normal/images/Normal-4379.png,Normal/masks/Normal-4379.png,True,G13362,train,b185d595fc9bc556edb667cfc23a83a2,d979c61a1d451b59
Normal-438.png,Normal,Normal/images/Normal-438.png,Normal/masks/Normal-438.png,True,G13363,train,9324b7798840435028aa92e11e7710eb,d068c31dcce638d7
Normal-4380.png,Normal,Normal/images/Normal-4380.png,Normal/masks/Normal-4380.png,True,G13364,train,5be4c907f07c45aa11239522a5f17417,d03d4a0b3c2d2f3d
Normal-4381.png,Normal,Normal/images/Normal-4381.png,Normal/masks/Normal-4381.png,True,G13365,val,ea8e5885d998395a1c65fe381921357e,d87f4718453a1d0f
Normal-4382.png,Normal,Normal/images/Normal-4382.png,Normal/masks/Normal-4382.png,True,G13366,val,fbaf34d71151a92090ebcd3840799e58,d42fc35313431f1e
Normal-4383.png,Normal,Normal/images/Normal-4383.png,Normal/masks/Normal-4383.png,True,G13367,test,54d8038d587cc1456150a01ffc0a8935,d43c83430f8badbc
Normal-4384.png,Normal,Normal/images/Normal-4384.png,Normal/masks/Normal-4384.png,True,G13368,train,b95b4d9b00ecdb46c1cfcca9e91d55d7,817b16161a5ff85a
Normal-4385.png,Normal,Normal/images/Normal-4385.png,Normal/masks/Normal-4385.png,True,G13369,train,545599e91d000a1c5d3d17ac6dd2beeb,9d7dc25a415b1b49
Normal-4386.png,Normal,Normal/images/Normal-4386.png,Normal/masks/Normal-4386.png,True,G13370,train,6955d6587e68ceb6129ff957f0ff78bb,81695e1c5a7be863
Normal-4387.png,Normal,Normal/images/Normal-4387.png,Normal/masks/Normal-4387.png,True,G13371,test,731255d48613293a9eb19e9898c60623,d928461a1a4d39ff
Normal-4388.png,Normal,Normal/images/Normal-4388.png,Normal/masks/Normal-4388.png,True,G13372,train,76369ded922b708666b26d84d4f19bf3,903b5a1f383fe445
Normal-4389.png,Normal,Normal/images/Normal-4389.png,Normal/masks/Normal-4389.png,True,G13373,val,b16607b26683ce125574c22bbdb44262,c829c71f1e561ab9
Normal-439.png,Normal,Normal/images/Normal-439.png,Normal/masks/Normal-439.png,True,G13374,train,0f221bb9698e771c9dd87aad58ba37d6,91bec61a8c0d2d9f
Normal-4390.png,Normal,Normal/images/Normal-4390.png,Normal/masks/Normal-4390.png,True,G13375,train,005b7ce6d6fe392db7aaec957717e063,c83dc61a1e475e1d
Normal-4391.png,Normal,Normal/images/Normal-4391.png,Normal/masks/Normal-4391.png,True,G13376,train,efcff34a9d53943022b15eb44530b4dc,817897861f1bca1f
Normal-4392.png,Normal,Normal/images/Normal-4392.png,Normal/masks/Normal-4392.png,True,G13377,train,a29cac28e3958a7dd88bec74e64a1ce8,c0bc2f8d3cdc30f2
Normal-4393.png,Normal,Normal/images/Normal-4393.png,Normal/masks/Normal-4393.png,True,G13378,train,1093a15211291527ad21dbcbf663629c,d42a435f4a2f1c2f
Normal-4394.png,Normal,Normal/images/Normal-4394.png,Normal/masks/Normal-4394.png,True,G13379,train,1f6942684fe4cdb5db5a4c1471028be8,917a561a191f3a2f
Normal-4395.png,Normal,Normal/images/Normal-4395.png,Normal/masks/Normal-4395.png,True,G13380,train,9aadfbd612ca4778abb04afcaad5a019,dc2d03495c6e79f0
Normal-4396.png,Normal,Normal/images/Normal-4396.png,Normal/masks/Normal-4396.png,True,G13381,train,73874cba985e68d9be5b46d5cefac309,885f061e5a5d5a79
Normal-4397.png,Normal,Normal/images/Normal-4397.png,Normal/masks/Normal-4397.png,True,G13382,test,d6fa81ab2025479559b12b31a5a62d6b,893f861a984df8f1
Normal-4398.png,Normal,Normal/images/Normal-4398.png,Normal/masks/Normal-4398.png,True,G13383,train,cb61af49e7c0f4da8fd136f7be8f5809,cc3cd24bd24f5e50
Normal-4399.png,Normal,Normal/images/Normal-4399.png,Normal/masks/Normal-4399.png,True,G13384,test,cdcdd59c5fb96496573999d67ec010ee,d939c65aba447e18
Normal-44.png,Normal,Normal/images/Normal-44.png,Normal/masks/Normal-44.png,True,G13385,train,75400d0dfa7d5556fd3913bd7de1952c,dd2dc7031bc51a1d
Normal-440.png,Normal,Normal/images/Normal-440.png,Normal/masks/Normal-440.png,True,G13386,val,0ff972babd0f9efc3d01f8464227f6b7,913ae27a78783959
Normal-4400.png,Normal,Normal/images/Normal-4400.png,Normal/masks/Normal-4400.png,True,G13387,val,5d0ed8b98dacb60b02eeb39fc5982359,dd12c30f0d2a3d97
Normal-4401.png,Normal,Normal/images/Normal-4401.png,Normal/masks/Normal-4401.png,True,G13388,train,7ab5cdb3e3e16f934892dd21a2f5a13c,dd2fd25f0c0a2d38
Normal-4402.png,Normal,Normal/images/Normal-4402.png,Normal/masks/Normal-4402.png,True,G13389,train,63c29f573ef6969a501448cf2745b78a,dd2cc24b1c4f2abc
Normal-4403.png,Normal,Normal/images/Normal-4403.png,Normal/masks/Normal-4403.png,True,G13390,train,49fcb17e8020b6355cec84fa44a640dd,d43d43575f4a0e1c
Normal-4404.png,Normal,Normal/images/Normal-4404.png,Normal/masks/Normal-4404.png,True,G13391,train,2f20db8a54bce442634ca05f581990e6,d56806c61b97c70f
Normal-4405.png,Normal,Normal/images/Normal-4405.png,Normal/masks/Normal-4405.png,True,G13392,train,0555d06d0facbf818b41df561fbebdbb,9920861a293f7f1f
Normal-4406.png,Normal,Normal/images/Normal-4406.png,Normal/masks/Normal-4406.png,True,G13393,train,1cd3e8b2619fde7594f8b101e599056f,816f861a1b5d7959
Normal-4407.png,Normal,Normal/images/Normal-4407.png,Normal/masks/Normal-4407.png,True,G13394,train,d0185cf7ca9347a6c243f1de2c3805b6,dc2fc24f0e8e0b1e
Normal-4408.png,Normal,Normal/images/Normal-4408.png,Normal/masks/Normal-4408.png,True,G13395,train,38fe7441b85712aab1450b3ae0c7ade9,d93f465a194f4a38
Normal-4409.png,Normal,Normal/images/Normal-4409.png,Normal/masks/Normal-4409.png,True,G13396,train,58e6ba9e5d6c5ebf83d43de916203a3d,dc2c06031f5e5b79
Normal-441.png,Normal,Normal/images/Normal-441.png,Normal/masks/Normal-441.png,True,G13397,train,103b8f103eafd5bec2ab216232989887,912ad65a5c597971
Normal-4410.png,Normal,Normal/images/Normal-4410.png,Normal/masks/Normal-4410.png,True,G13398,train,020dd1b932a44ee84334e87a1d0ae121,cc3cc24b1fc22f9c
Normal-4411.png,Normal,Normal/images/Normal-4411.png,Normal/masks/Normal-4411.png,True,G13399,train,58936edd29b2847ef08e359cd09da796,f949c31a09473a7d
Normal-4412.png,Normal,Normal/images/Normal-4412.png,Normal/masks/Normal-4412.png,True,G13400,train,c711b7f1b251c91cfe94bdfb23e047e6,9d6ec65249c51aad
Normal-4413.png,Normal,Normal/images/Normal-4413.png,Normal/masks/Normal-4413.png,True,G13401,val,a60f40772e84a00f03b9ae0b120c6b13,dc3dd24b1a524b1e
Normal-4414.png,Normal,Normal/images/Normal-4414.png,Normal/masks/Normal-4414.png,True,G13402,train,abc2114bb773c3ccc6e9dae7107dfa1e,917b565e164d1b38
Normal-4415.png,Normal,Normal/images/Normal-4415.png,Normal/masks/Normal-4415.png,True,G13403,val,f9979a80a00549dd65174eb3ef8ab3da,9c6e075a4f686ce8
Normal-4416.png,Normal,Normal/images/Normal-4416.png,Normal/masks/Normal-4416.png,True,G13404,val,012f590acb81c1b992030962e75ff6c5,997f165a4f5a4918
Normal-4417.png,Normal,Normal/images/Normal-4417.png,Normal/masks/Normal-4417.png,True,G13405,test,cd280845ff6152e920c7f3b2f8e040dd,dd3cc25b1f405d19
Normal-4418.png,Normal,Normal/images/Normal-4418.png,Normal/masks/Normal-4418.png,True,G13406,test,46666b54581b8a5a60dfc8629a861b95,887f861e151f1f45
Normal-4419.png,Normal,Normal/images/Normal-4419.png,Normal/masks/Normal-4419.png,True,G13407,train,33c5b3fb2205c79855c2d353c42e4242,8978069e5e495e5b
Normal-442.png,Normal,Normal/images/Normal-442.png,Normal/masks/Normal-442.png,True,G13408,test,7e89637b5115d96e439f5f7a0be9b619,982fc71acb493c69
Normal-4420.png,Normal,Normal/images/Normal-4420.png,Normal/masks/Normal-4420.png,True,G13409,train,35acc139241340b3cc709b134e79a528,817987861b590f5f
Normal-4421.png,Normal,Normal/images/Normal-4421.png,Normal/masks/Normal-4421.png,True,G13410,val,c40e6655faf8c4ffe89848d04c90aeac,c93b07461f15564f
Normal-4422.png,Normal,Normal/images/Normal-4422.png,Normal/masks/Normal-4422.png,True,G13411,train,d573a529ff60c0f45078dda8ddce1e27,c03ec713194f5d5c
Normal-4423.png,Normal,Normal/images/Normal-4423.png,Normal/masks/Normal-4423.png,True,G13412,train,ab9526ebc19c4673adcc5e01d4356fdc,c87d425b544f1d0f
Normal-4424.png,Normal,Normal/images/Normal-4424.png,Normal/masks/Normal-4424.png,True,G13413,train,2b959717f5851b3ffd7b525d17a547cf,996c065a4d6d7ae1
Normal-4425.png,Normal,Normal/images/Normal-4425.png,Normal/masks/Normal-4425.png,True,G13414,train,8ebd4892d464d0bb2502dde36d42916f,d93dc406192d5f3c
Normal-4426.png,Normal,Normal/images/Normal-4426.png,Normal/masks/Normal-4426.png,True,G13415,train,e22042e50c0d61510bc97dff849ed5c1,dd78430b5d293d1c
Normal-4427.png,Normal,Normal/images/Normal-4427.png,Normal/masks/Normal-4427.png,True,G13416,train,96a524f1b002f8253657c5c4b187b499,cc34030f0bce7ced
Normal-4428.png,Normal,Normal/images/Normal-4428.png,Normal/masks/Normal-4428.png,True,G13417,train,01a37f8f10407866b978cd33e66a871d,cc3c434b4f4ce47c
Normal-4429.png,Normal,Normal/images/Normal-4429.png,Normal/masks/Normal-4429.png,True,G13418,train,3efddceae71c59cdf02f5346de2878e9,dd3d421f0f282db8
Normal-443.png,Normal,Normal/images/Normal-443.png,Normal/masks/Normal-443.png,True,G13419,val,0cb00212e39db696fe895614d4a21b56,d17cc65a8c4d3ccc
Normal-4430.png,Normal,Normal/images/Normal-4430.png,Normal/masks/Normal-4430.png,True,G13420,val,1b2dd17a133e63c0aa4d7598d4d7ed5a,c97d571a554b1c49
Normal-4431.png,Normal,Normal/images/Normal-4431.png,Normal/masks/Normal-4431.png,True,G13421,test,fc65eb60330664265a2a35002a4036e1,c8354746db4a6c79
Normal-4432.png,Normal,Normal/images/Normal-4432.png,Normal/masks/Normal-4432.png,True,G13422,test,236ce61b30b12dc2a281b03b6d8cbf42,817f0612175f195f
Normal-4433.png,Normal,Normal/images/Normal-4433.png,Normal/masks/Normal-4433.png,True,G13423,test,0c810884db7d0342b1ebe95ea1db08b7,997fc45a5d485a19
Normal-4434.png,Normal,Normal/images/Normal-4434.png,Normal/masks/Normal-4434.png,True,G13424,test,c0449fd2727397b8887e34999ae069b5,dc3dc2559342af52
Normal-4435.png,Normal,Normal/images/Normal-4435.png,Normal/masks/Normal-4435.png,True,G13425,train,3617f4ada04c741376dcf460f593f73d,dc3cc2531fc50b0f
Normal-4436.png,Normal,Normal/images/Normal-4436.png,Normal/masks/Normal-4436.png,True,G13426,train,7f8f4c9d4d1b1622f834f538869d1d01,dd2e424f0f454f2c
Normal-4437.png,Normal,Normal/images/Normal-4437.png,Normal/masks/Normal-4437.png,True,G13427,val,997bbe173559f59718f01bbb330a282e,9979479658c75e90
Normal-4438.png,Normal,Normal/images/Normal-4438.png,Normal/masks/Normal-4438.png,True,G13428,train,c789c9b9adb22028654e9321a8a10889,c13c0b0f3cd374d3
Normal-4439.png,Normal,Normal/images/Normal-4439.png,Normal/masks/Normal-4439.png,True,G13429,train,eb6342309b0129b70f41512ba0b30efe,817885061e5b5b7f
Normal-444.png,Normal,Normal/images/Normal-444.png,Normal/masks/Normal-444.png,True,G13430,train,abe61915bd7cdebdf445467db5673cc1,9129e61ed84b3c6d
Normal-4440.png,Normal,Normal/images/Normal-4440.png,Normal/masks/Normal-4440.png,True,G13431,train,17b3e64804fab0000ec6ce19878ff6cd,d92c461b056f1caf
Normal-4441.png,Normal,Normal/images/Normal-4441.png,Normal/masks/Normal-4441.png,True,G13432,train,f870ee8c6e283a15df45704c0c2af744,d92b561f1b0d3a38
Normal-4442.png,Normal,Normal/images/Normal-4442.png,Normal/masks/Normal-4442.png,True,G13433,train,d6c8e9cd33f4eed75c4e0bf2bcf69b0f,b97dc6561b4d4c18
Normal-4443.png,Normal,Normal/images/Normal-4443.png,Normal/masks/Normal-4443.png,True,G13434,train,39580b580e5e42f3cf5f43e45a8e86eb,d1b9c61f90473f50
Normal-4444.png,Normal,Normal/images/Normal-4444.png,Normal/masks/Normal-4444.png,True,G13435,train,90949b479e407d6f2b1823691784d2a8,dc2cd6d34b0dca4a
Normal-4445.png,Normal,Normal/images/Normal-4445.png,Normal/masks/Normal-4445.png,True,G13436,train,ed45ed11340adb22cf23e6e9184f3096,996ad71a5543594e
Normal-4446.png,Normal,Normal/images/Normal-4446.png,Normal/masks/Normal-4446.png,True,G13437,train,7ce3b7f20b8dab7d264fa901293544f4,cd6e461e594d2e1c
Normal-4447.png,Normal,Normal/images/Normal-4447.png,Normal/masks/Normal-4447.png,True,G13438,train,55d35154f030e29362098c0abd1ae901,d969c61e954e5a46
Normal-4448.png,Normal,Normal/images/Normal-4448.png,Normal/masks/Normal-4448.png,True,G13439,train,aa3032bd980994723551ebda7c5af262,dd3c421bcc0e1ceb
Normal-4449.png,Normal,Normal/images/Normal-4449.png,Normal/masks/Normal-4449.png,True,G13440,train,7a7ef96665313703fe70625ff00eb4ed,d47c43530f4e0f4e
Normal-445.png,Normal,Normal/images/Normal-445.png,Normal/masks/Normal-445.png,True,G13441,train,3e86ee7e447504bafb3843d0ab87bbb6,956e0f0b455c4cf5
Normal-4450.png,Normal,Normal/images/Normal-4450.png,Normal/masks/Normal-4450.png,True,G13442,train,8cbef7e853fc1ae376664ca56e6dc329,d43d52439b1a4f4e
Normal-4451.png,Normal,Normal/images/Normal-4451.png,Normal/masks/Normal-4451.png,True,G13443,test,ba500e41be5c585bf123b88a4701230c,d979c51716461b1d
Normal-4452.png,Normal,Normal/images/Normal-4452.png,Normal/masks/Normal-4452.png,True,G13444,train,d40b1883d4fdaa67f09c8aa044a39859,897d461e1f4a1f1c
Normal-4453.png,Normal,Normal/images/Normal-4453.png,Normal/masks/Normal-4453.png,True,G13445,val,ce1f4a1b2e1bef7e516d109b6ebb48a3,9c74421bc75ac34f
Normal-4454.png,Normal,Normal/images/Normal-4454.png,Normal/masks/Normal-4454.png,True,G13446,val,55a07c7704cf8d72b41a0934b975d396,dc2f421f424f1c4f
Normal-4455.png,Normal,Normal/images/Normal-4455.png,Normal/masks/Normal-4455.png,True,G13447,test,00d459221dbe877376e9c919e08d8af1,dc2cc34b0e8e2b3d
Normal-4456.png,Normal,Normal/images/Normal-4456.png,Normal/masks/Normal-4456.png,True,G13448,val,bcb7897df2becc14867766d0873fa003,d97f47181a4d3839
Normal-4457.png,Normal,Normal/images/Normal-4457.png,Normal/masks/Normal-4457.png,True,G13449,train,7cb972a9ef409aa400ab02a65dea5c0c,dc3d53430b432f2d
Normal-4458.png,Normal,Normal/images/Normal-4458.png,Normal/masks/Normal-4458.png,True,G13450,train,937320576f1d96e8f93ab6ac0f234d1f,816906165f7eb86b
Normal-4459.png,Normal,Normal/images/Normal-4459.png,Normal/masks/Normal-4459.png,True,G13451,train,be728fa3af8f471cc5fce873f2b0fc9b,dc3d431b4c4e396c
Normal-446.png,Normal,Normal/images/Normal-446.png,Normal/masks/Normal-446.png,True,G13452,train,f4386db3474658ddb828b7accc5314f5,9139e65a986d3939
Normal-4460.png,Normal,Normal/images/Normal-4460.png,Normal/masks/Normal-4460.png,True,G13453,train,515ac5acddc003aa25d5655a27e01bf8,822cefc7894ae0f3
Normal-4461.png,Normal,Normal/images/Normal-4461.png,Normal/masks/Normal-4461.png,True,G13454,train,3759c966f91cde8ca06231741ba8d977,dd2dd5521f455819
Normal-4462.png,Normal,Normal/images/Normal-4462.png,Normal/masks/Normal-4462.png,True,G13455,train,2a0a2da2b22ea8a479e1c63952b0daa3,997f061a59185f1e
Normal-4463.png,Normal,Normal/images/Normal-4463.png,Normal/masks/Normal-4463.png,True,G13456,val,de044df7d30c11cb05c54356bad9774f,807f4e1c4d6d94e5
Normal-4464.png,Normal,Normal/images/Normal-4464.png,Normal/masks/Normal-4464.png,True,G13457,train,ad1b82c32bff9e7ea2a0af205057c657,bd0c021b5f4e5c79
Normal-4465.png,Normal,Normal/images/Normal-4465.png,Normal/masks/Normal-4465.png,True,G13458,test,76d60fa21c0c112a40040234b1051d66,993906185e6b79e5
Normal-4466.png,Normal,Normal/images/Normal-4466.png,Normal/masks/Normal-4466.png,True,G13459,test,065b72d58f2d97d20f9a69a302ed82da,893b06965ecc7ad8
Normal-4467.png,Normal,Normal/images/Normal-4467.png,Normal/masks/Normal-4467.png,True,G13460,test,3ee1c635ed96c838931ab1c756e40d45,8979851e1a5d5b1b
Normal-4468.png,Normal,Normal/images/Normal-4468.png,Normal/masks/Normal-4468.png,True,G13461,train,caf789d630a9a194f35357535f0c1509,992f861a194f78f1
Normal-4469.png,Normal,Normal/images/Normal-4469.png,Normal/masks/Normal-4469.png,True,G13462,val,3ef86372772336f7b2da31858daf94e6,d13c0e19574f78e8
Normal-447.png,Normal,Normal/images/Normal-447.png,Normal/masks/Normal-447.png,True,G13463,train,c82b8b5cc5150bd25aac9883975b01b4,9129e61ad8793dca
Normal-4470.png,Normal,Normal/images/Normal-4470.png,Normal/masks/Normal-4470.png,True,G13464,train,8e557c989b3c020632aeccc536d94fd5,b879c5565e54551c
Normal-4471.png,Normal,Normal/images/Normal-4471.png,Normal/masks/Normal-4471.png,True,G13465,train,7b2a21a7cea39b47b2b87bc4ac422c88,993d464b194eaab9
Normal-4472.png,Normal,Normal/images/Normal-4472.png,Normal/masks/Normal-4472.png,True,G13466,train,3b1b559bf05685e32b43bac99f376d55,d03d4a575b1b4c4d
Normal-4473.png,Normal,Normal/images/Normal-4473.png,Normal/masks/Normal-4473.png,True,G13467,train,f0fa57e9acd4c609b6db321e24507bbb,817906061a5f5f5f
Normal-4474.png,Normal,Normal/images/Normal-4474.png,Normal/masks/Normal-4474.png,True,G13468,train,6efc1248e0051402e962567346a9809a,897b571a58595959
Normal-4475.png,Normal,Normal/images/Normal-4475.png,Normal/masks/Normal-4475.png,True,G13469,val,51a9af1b5077b3cc62178b5fd5f6228f,9c2cc20f0f5d0f1f
Normal-4476.png,Normal,Normal/images/Normal-4476.png,Normal/masks/Normal-4476.png,True,G13470,train,709656afc333e5ce8b0ee3a101258f2a,dd2dc25b43c30bd1
Normal-4477.png,Normal,Normal/images/Normal-4477.png,Normal/masks/Normal-4477.png,True,G13471,train,5bc46ce9124934e068100c6d7dffa8b3,897d061e5af86a69
Normal-4478.png,Normal,Normal/images/Normal-4478.png,Normal/masks/Normal-4478.png,True,G13472,train,b32e33401d30092256c94d2a7b9d482c,9979061e5a5c5a79
Normal-4479.png,Normal,Normal/images/Normal-4479.png,Normal/masks/Normal-4479.png,True,G13473,train,e500f5df832b188203c3ffe0d0e33293,993ec65b1e4c6b18
Normal-448.png,Normal,Normal/images/Normal-448.png,Normal/masks/Normal-448.png,True,G13474,train,384685699721e9fda1a497e3f22786f0,816be67ae9692945
Normal-4480.png,Normal,Normal/images/Normal-4480.png,Normal/masks/Normal-4480.png,True,G13475,train,7306993a79d019fd4ec567b543aa8fd6,9138021f5b697b5b
Normal-4481.png,Normal,Normal/images/Normal-4481.png,Normal/masks/Normal-4481.png,True,G13476,train,330d158fe10cef46eb38269e7ca106fa,817b045e5a78ede8
Normal-4482.png,Normal,Normal/images/Normal-4482.png,Normal/masks/Normal-4482.png,True,G13477,train,10e762ea09e140b96a08096abc876ae5,893b571a145b1f4d
Normal-4483.png,Normal,Normal/images/Normal-4483.png,Normal/masks/Normal-4483.png,True,G13478,train,8a1c6000d3495db9d9d9944469a69368,912986151f1e3b5f
Normal-4484.png,Normal,Normal/images/Normal-4484.png,Normal/masks/Normal-4484.png,True,G13479,train,28c00bac382cc33bba5e78ab396f959d,9cbd46461bc7461e
Normal-4485.png,Normal,Normal/images/Normal-4485.png,Normal/masks/Normal-4485.png,True,G13480,train,d2bc8b532b93832ff43a5f3259cf7969,897986565b0dfa19
Normal-4486.png,Normal,Normal/images/Normal-4486.png,Normal/masks/Normal-4486.png,True,G13481,val,6fb9132aa76c1d65ad1a1884cf7f1656,d17f865a144f585c
Normal-4487.png,Normal,Normal/images/Normal-4487.png,Normal/masks/Normal-4487.png,True,G13482,train,77e427e30448f11fbd9c538079e07c3b,89fdc61ec41ec50e
Normal-4488.png,Normal,Normal/images/Normal-4488.png,Normal/masks/Normal-4488.png,True,G13483,train,6024c9a3993a5248ff79691f26d04ff8,997d061e045ebbc5
Normal-4489.png,Normal,Normal/images/Normal-4489.png,Normal/masks/Normal-4489.png,True,G13484,val,fdb2818a9a2cb25fd1c95978e7f4032f,913b06197bede8a1
Normal-449.png,Normal,Normal/images/Normal-449.png,Normal/masks/Normal-449.png,True,G13485,test,19487a69e501672b74c636e0794554a0,9139c61a79693979
Normal-4490.png,Normal,Normal/images/Normal-4490.png,Normal/masks/Normal-4490.png,True,G13486,test,5950db20c8c30fad0ca15f1b620fbf59,d17d46101b4f3b39
Normal-4491.png,Normal,Normal/images/Normal-4491.png,Normal/masks/Normal-4491.png,True,G13487,test,a53c9655bc00c9c8961fcaa0556cfdd4,dd3dc21b462f1ac4
Normal-4492.png,Normal,Normal/images/Normal-4492.png,Normal/masks/Normal-4492.png,True,G13488,train,0a57ba345f7a9298b9fb03437adb5008,8369041e195f7b5b
Normal-4493.png,Normal,Normal/images/Normal-4493.png,Normal/masks/Normal-4493.png,True,G13489,train,56f19e0bf43b30a7acafbcacaeff529f,882a465a1a5f5f5d
Normal-4494.png,Normal,Normal/images/Normal-4494.png,Normal/masks/Normal-4494.png,True,G13490,train,435de05b859e8157de58e3ede0908f33,d96a865f0a1e5b1a
Normal-4495.png,Normal,Normal/images/Normal-4495.png,Normal/masks/Normal-4495.png,True,G13491,train,0703dbdfa5bdceb7afa986dcfcfb380e,d97c86431f4c4e59
Normal-4496.png,Normal,Normal/images/Normal-4496.png,Normal/masks/Normal-4496.png,True,G13492,train,a685042cc621fa0aab390ee5d5106128,917b069a1b2dea1d
Normal-4497.png,Normal,Normal/images/Normal-4497.png,Normal/masks/Normal-4497.png,True,G13493,train,74a926d4d25948de2eeafeaffd730d07,d938860b1f4d393e
Normal-4498.png,Normal,Normal/images/Normal-4498.png,Normal/masks/Normal-4498.png,True,G13494,train,b0ad03eeb9e9b1c47f87549ecdd32f56,dd0a475b0c2f18ed
Normal-4499.png,Normal,Normal/images/Normal-4499.png,Normal/masks/Normal-4499.png,True,G13495,val,6c150265cafcd87a7f4372c28314996a,897b85161a4d4f1f
Normal-45.png,Normal,Normal/images/Normal-45.png,Normal/masks/Normal-45.png,True,G13496,train,0ac56d1d11bb8383f6a2ab8dd6b6b7d0,990d861e4d5e1f4d
Normal-450.png,Normal,Normal/images/Normal-450.png,Normal/masks/Normal-450.png,True,G13497,train,8974a77270477bef1df051a90de8d507,d42fcb1a00577ce5
Normal-4500.png,Normal,Normal/images/Normal-4500.png,Normal/masks/Normal-4500.png,True,G13498,train,3fb87ab26434a0e6a809f86c93b976a8,dd3d42431f1c4b1e
Normal-4501.png,Normal,Normal/images/Normal-4501.png,Normal/masks/Normal-4501.png,True,G13499,test,4f28acccc623191a82d324e5550652e3,993dc41e154e1afc
Normal-4502.png,Normal,Normal/images/Normal-4502.png,Normal/masks/Normal-4502.png,True,G13500,train,9526ba372503c276099bb106b4fd9610,dd28d21bd54a1f45
Normal-4503.png,Normal,Normal/images/Normal-4503.png,Normal/masks/Normal-4503.png,True,G13501,train,b7f59eb13743390a48286a33e07b0fc6,d93d460f1a0e693b
Normal-4504.png,Normal,Normal/images/Normal-4504.png,Normal/masks/Normal-4504.png,True,G13502,train,db32949b4ded45a8a28cc2c08e4bf08b,993fc61ad4469a4e
Normal-4505.png,Normal,Normal/images/Normal-4505.png,Normal/masks/Normal-4505.png,True,G13503,test,1ca1b49eaa286bad11607152493676a4,c96d861a055e39cf
Normal-4506.png,Normal,Normal/images/Normal-4506.png,Normal/masks/Normal-4506.png,True,G13504,train,3ab6656d40836938fdce2817e8b0f246,dc2e434b0e4a6d3e
Normal-4507.png,Normal,Normal/images/Normal-4507.png,Normal/masks/Normal-4507.png,True,G13505,train,d90c62b5bebdf289c0481a0c0a84a738,d939c615144f1f1b
Normal-4508.png,Normal,Normal/images/Normal-4508.png,Normal/masks/Normal-4508.png,True,G13506,val,8184683ecf30526c90359fe51cfa168f,c97b15061e497a3b
Normal-4509.png,Normal,Normal/images/Normal-4509.png,Normal/masks/Normal-4509.png,True,G13507,val,bba210b47f89f350c6f51ec6056ace31,9979c65e18475a39
Normal-451.png,Normal,Normal/images/Normal-451.png,Normal/masks/Normal-451.png,True,G13508,test,829303aebc95a085514aeb55520753cb,917f871a8b1b18cd
Normal-4510.png,Normal,Normal/images/Normal-4510.png,Normal/masks/Normal-4510.png,True,G13509,train,278e125efbab4c70edd2021e35b058da,897f871e151f1e09
Normal-4511.png,Normal,Normal/images/Normal-4511.png,Normal/masks/Normal-4511.png,True,G13510,train,022c6d09f9d6aa93c081098499e93969,9d6ed25f4d151868
Normal-4512.png,Normal,Normal/images/Normal-4512.png,Normal/masks/Normal-4512.png,True,G13511,train,470c0ea6dba6f362fa81cd7e0caa4f30,dd2d424b0b1e4b2f
Normal-4513.png,Normal,Normal/images/Normal-4513.png,Normal/masks/Normal-4513.png,True,G13512,train,d49c5515c338a0b61197bd2286d016cc,9968161e5a5a79e9
Normal-4514.png,Normal,Normal/images/Normal-4514.png,Normal/masks/Normal-4514.png,True,G13513,train,b901ad51796b26f1a2ce6bcb332f59e8,997fc61a055a39c6
Normal-4515.png,Normal,Normal/images/Normal-4515.png,Normal/masks/Normal-4515.png,True,G13514,train,fce97cbe7af5962f3e6687e5899d6096,916e461e11577979
Normal-4516.png,Normal,Normal/images/Normal-4516.png,Normal/masks/Normal-4516.png,True,G13515,train,d0607bb81f1e1acb09ad3f8d8be3bca4,81695e46195b5e4f
Normal-4517.png,Normal,Normal/images/Normal-4517.png,Normal/masks/Normal-4517.png,True,G13516,train,d231c75f30ff9c6a5ee15490e0ed9d7e,c979c61a1b4f1999
Normal-4518.png,Normal,Normal/images/Normal-4518.png,Normal/masks/Normal-4518.png,True,G13517,train,e41508daf9724bdbfffd6109a8ac4f9e,982dc51e5c6d587a
Normal-4519.png,Normal,Normal/images/Normal-4519.png,Normal/masks/Normal-4519.png,True,G13518,train,08f07ce977c232dbed4915279abcd320,c03d6b433da235f4
Normal-452.png,Normal,Normal/images/Normal-452.png,Normal/masks/Normal-452.png,True,G13519,train,0224c97e381e9eb03353981e034fc3b8,9129c75ade6b3948
Normal-4520.png,Normal,Normal/images/Normal-4520.png,Normal/masks/Normal-4520.png,True,G13520,val,d535e09202585c31f0968da01ec94847,cc5cd30f4e0c1dad
Normal-4521.png,Normal,Normal/images/Normal-4521.png,Normal/masks/Normal-4521.png,True,G13521,val,889b3dcdb9d340699cc25f4b038fc284,dd5942521b4f4d1b
Normal-4522.png,Normal,Normal/images/Normal-4522.png,Normal/masks/Normal-4522.png,True,G13522,train,20d880edb4492326dec5afcbba1f9af0,887f061a595f1b0f
Normal-4523.png,Normal,Normal/images/Normal-4523.png,Normal/masks/Normal-4523.png,True,G13523,train,d987a36f3652e3f068da595b14df07cd,dd3d524fd04b1f02
Normal-4524.png,Normal,Normal/images/Normal-4524.png,Normal/masks/Normal-4524.png,True,G13524,val,18e19af14d15cd8f1f2485d00be39753,c03d12434b3ff8e9
Normal-4525.png,Normal,Normal/images/Normal-4525.png,Normal/masks/Normal-4525.png,True,G13525,test,fb079a3630e7ed120452eb81e04d666e,9129c61a5d5f7943
Normal-4526.png,Normal,Normal/images/Normal-4526.png,Normal/masks/Normal-4526.png,True,G13526,test,9acfdfeaa94786b16a452f2413a42199,c82f520f164f1f5c
Normal-4527.png,Normal,Normal/images/Normal-4527.png,Normal/masks/Normal-4527.png,True,G13527,test,e759a615af9b76a89f328eba00386e2b,9178061e1a5f5b79
Normal-4528.png,Normal,Normal/images/Normal-4528.png,Normal/masks/Normal-4528.png,True,G13528,train,c77877a9ff4131f9a0f617ee7f0be848,dd3d82021d4eed39
Normal-4529.png,Normal,Normal/images/Normal-4529.png,Normal/masks/Normal-4529.png,True,G13529,train,84b64e5a473edc736108c014a6ced4a9,817b06081f5f5b5b
Normal-453.png,Normal,Normal/images/Normal-453.png,Normal/masks/Normal-453.png,True,G13530,train,31ac393e87ceada342c4cc5514b3837b,9029c71e9e6a39e9
Normal-4530.png,Normal,Normal/images/Normal-4530.png,Normal/masks/Normal-4530.png,True,G13531,train,3fb401ebd6397185f3fee7f79f62a1b8,9179c6061a1fee1b
Normal-4531.png,Normal,Normal/images/Normal-4531.png,Normal/masks/Normal-4531.png,True,G13532,train,0dcbb7e92518ce69658817ec5f5224b6,d52f42575b444f1c
Normal-4532.png,Normal,Normal/images/Normal-4532.png,Normal/masks/Normal-4532.png,True,G13533,train,ed7c6ccdb62c7b886b91eee1b905f178,dcbdd3039fc24618
Normal-4533.png,Normal,Normal/images/Normal-4533.png,Normal/masks/Normal-4533.png,True,G13534,test,0165b464e67a4cb1049b2489abac0098,dc3dd3490c1e292f
Normal-4534.png,Normal,Normal/images/Normal-4534.png,Normal/masks/Normal-4534.png,True,G13535,train,b404e7e490405b7a3af3b9f0b3394c15,897b07161e175a1f
Normal-4535.png,Normal,Normal/images/Normal-4535.png,Normal/masks/Normal-4535.png,True,G13536,train,31853efd0c18e3fcbeba635699e1ed7f,cc3dd24b0bc20f1f
Normal-4536.png,Normal,Normal/images/Normal-4536.png,Normal/masks/Normal-4536.png,True,G13537,train,a4496e2d83c1c16b7100d35884e3a0a5,837a5602533b3b3b
Normal-4537.png,Normal,Normal/images/Normal-4537.png,Normal/masks/Normal-4537.png,True,G13538,train,f0f4bbc1a913dee2e61b8d1fdfcb616d,c17b52525a545b5b
Normal-4538.png,Normal,Normal/images/Normal-4538.png,Normal/masks/Normal-4538.png,True,G13539,train,e72a3763429111aee405a1d5fd97cc03,d43dc3431d0e6e3c
Normal-4539.png,Normal,Normal/images/Normal-4539.png,Normal/masks/Normal-4539.png,True,G13540,train,62eec28c4f479bdd97071ca30b478d0f,cd0cc6c21b4f4d4f
Normal-454.png,Normal,Normal/images/Normal-454.png,Normal/masks/Normal-454.png,True,G13541,test,e2ed954ff24822e058c66f786ab56684,d52dc30fccc81cd3
Normal-4540.png,Normal,Normal/images/Normal-4540.png,Normal/masks/Normal-4540.png,True,G13542,train,34c19900ff0edb8a3490bbdee9ea68b7,d971421ec61ad55b
Normal-4541.png,Normal,Normal/images/Normal-4541.png,Normal/masks/Normal-4541.png,True,G13543,train,fe877f904802c222dc263cabc096f06d,987b05c61e5a5b1d
Normal-4542.png,Normal,Normal/images/Normal-4542.png,Normal/masks/Normal-4542.png,True,G13544,val,a2d8f04b9364236c2c1c9a882dbbb031,a96c86185d4d5dce
Normal-4543.png,Normal,Normal/images/Normal-4543.png,Normal/masks/Normal-4543.png,True,G13545,train,f3b8ff71e3b0acc921300095a536fe14,cc6c834b0f462d3f
Normal-4544.png,Normal,Normal/images/Normal-4544.png,Normal/masks/Normal-4544.png,True,G13546,val,d5f79b66b29b7c15c1f34f30c6834974,dc2f824b5d4c687c
Normal-4545.png,Normal,Normal/images/Normal-4545.png,Normal/masks/Normal-4545.png,True,G13547,train,e76abdfd3fb92b53422563f2cd6a3139,9969c71ed4561647
Normal-4546.png,Normal,Normal/images/Normal-4546.png,Normal/masks/Normal-4546.png,True,G13548,val,1347b3b667eb290bda7c2b91e04bb997,dd2c525353470b5b
Normal-4547.png,Normal,Normal/images/Normal-4547.png,Normal/masks/Normal-4547.png,True,G13549,test,e4aaf94012ab20d70749e87923c4d234,dd6fc4131b454e38
Normal-4548.png,Normal,Normal/images/Normal-4548.png,Normal/masks/Normal-4548.png,True,G13550,test,f1c7e4d00640b42a45211394636b482d,897b041e5a6d7578
Normal-4549.png,Normal,Normal/images/Normal-4549.png,Normal/masks/Normal-4549.png,True,G13551,test,6f7eedf6e3ad46ad41ba165856c5363e,917b0c4e1a78795b
Normal-455.png,Normal,Normal/images/Normal-455.png,Normal/masks/Normal-455.png,True,G13552,train,6f34bb34a0c15fc479cff0ff52d84b42,812bc65a78697979
Normal-4550.png,Normal,Normal/images/Normal-4550.png,Normal/masks/Normal-4550.png,True,G13553,train,2740de7361c96d1a3fe464a894e28438,9939c71a1b491d5b
Normal-4551.png,Normal,Normal/images/Normal-4551.png,Normal/masks/Normal-4551.png,True,G13554,train,df6da7051bb5242c4cd690122086bc22,dc2e530b431b0f2f
Normal-4552.png,Normal,Normal/images/Normal-4552.png,Normal/masks/Normal-4552.png,True,G13555,train,3468ba6fcb63e6a7e7114fc38d4f6741,d13d871a1ace2bb8
Normal-4553.png,Normal,Normal/images/Normal-4553.png,Normal/masks/Normal-4553.png,True,G13556,train,0158eaddab595d2ad814232b05799d42,dc3dc30f0e282c3f
Normal-4554.png,Normal,Normal/images/Normal-4554.png,Normal/masks/Normal-4554.png,True,G13557,train,b752e3fb2fde80fb218e9ce59b88d95c,884f065e5b5b4d78
Normal-4555.png,Normal,Normal/images/Normal-4555.png,Normal/masks/Normal-4555.png,True,G13558,train,535000e8eb9d39b8ac2e9f0da2a9de8e,981f065a49ebf1a5
Normal-4556.png,Normal,Normal/images/Normal-4556.png,Normal/masks/Normal-4556.png,True,G13559,train,64c5e57e8c804d635fe508d0d8abb111,dc2b531d4c285d2f
Normal-4557.png,Normal,Normal/images/Normal-4557.png,Normal/masks/Normal-4557.png,True,G13560,train,8d15e78d5a323251baf66200dc74e368,887946921a5def1b
Normal-4558.png,Normal,Normal/images/Normal-4558.png,Normal/masks/Normal-4558.png,True,G13561,train,d4c35f61327332376d652dbec9450b2c,ac7fc20b0f430f2e
Normal-4559.png,Normal,Normal/images/Normal-4559.png,Normal/masks/Normal-4559.png,True,G13562,train,b187c27dc1e439e6ffacc7667dab5223,d82ac2525f4d5e1b
Normal-456.png,Normal,Normal/images/Normal-456.png,Normal/masks/Normal-456.png,True,G13563,val,02a9d46962d8a5a30f26674cbbbb2e82,902bd25fdc5e1d30
Normal-4560.png,Normal,Normal/images/Normal-4560.png,Normal/masks/Normal-4560.png,True,G13564,val,902354ec3388d405a38375141f96d90a,d46cc34b0f0f0d3d
Normal-4561.png,Normal,Normal/images/Normal-4561.png,Normal/masks/Normal-4561.png,True,G13565,train,6c8a7fd4f300febcaf8f642adb2d0cca,802a475f1f4de8bc
Normal-4562.png,Normal,Normal/images/Normal-4562.png,Normal/masks/Normal-4562.png,True,G13566,val,295fc4fe0f81b75c5b2e4d49aec82845,d42d43439d88ecbd
Normal-4563.png,Normal,Normal/images/Normal-4563.png,Normal/masks/Normal-4563.png,True,G13567,test,7e8e898e8b782179aee3026d25d4fd8c,81e86e876a9e4e6a
Normal-4564.png,Normal,Normal/images/Normal-4564.png,Normal/masks/Normal-4564.png,True,G13568,test,fa08ac314bc50a37d5a0b7314a1e8e49,d12b571b0c493d2f
Normal-4565.png,Normal,Normal/images/Normal-4565.png,Normal/masks/Normal-4565.png,True,G13569,train,128b9895c078d952b03c03058fdc5eb3,fd6dc70e4c0a5c4a
Normal-4566.png,Normal,Normal/images/Normal-4566.png,Normal/masks/Normal-4566.png,True,G13570,train,866d804a99fba04c27c65307e820b8f8,c13997c61a1d5e4b
Normal-4567.png,Normal,Normal/images/Normal-4567.png,Normal/masks/Normal-4567.png,True,G13571,train,12ea22beaac292a669f4f8b09174da4c,8978061a1d4d79df
Normal-4568.png,Normal,Normal/images/Normal-4568.png,Normal/masks/Normal-4568.png,True,G13572,train,10c7b9ae2f834655216856c4007bc6bb,897bd65708455b1d
Normal-4569.png,Normal,Normal/images/Normal-4569.png,Normal/masks/Normal-4569.png,True,G13573,train,593e2f92b8154744538c38f596aae5e4,d91987461e2c7979
Normal-457.png,Normal,Normal/images/Normal-457.png,Normal/masks/Normal-457.png,True,G13574,train,6d6cdc7ec1b132f091d2aeb98317fb26,d47cc31b8dc93cc4
Normal-4570.png,Normal,Normal/images/Normal-4570.png,Normal/masks/Normal-4570.png,True,G13575,train,70a0b5fa5dcedd09146b7a1e93eaa7d6,d83b45571a38513f
Normal-4571.png,Normal,Normal/images/Normal-4571.png,Normal/masks/Normal-4571.png,True,G13576,train,f41e660ea3639e2efab1bd41f687c94b,d17b46461a8f4b99
Normal-4572.png,Normal,Normal/images/Normal-4572.png,Normal/masks/Normal-4572.png,True,G13577,train,efd2f1fb62a6c6f8061245df1fd54567,817d061e1b6b39f8
Normal-4573.png,Normal,Normal/images/Normal-4573.png,Normal/masks/Normal-4573.png,True,G13578,train,664d43dc7f492ddf7638a3dfe5cd636e,996986181f5f7ab0
Normal-4574.png,Normal,Normal/images/Normal-4574.png,Normal/masks/Normal-4574.png,True,G13579,test,8630be07a0a4311e053d444a1817ce85,897d861a1d5c7a4d
Normal-4575.png,Normal,Normal/images/Normal-4575.png,Normal/masks/Normal-4575.png,True,G13580,train,86525944475d530a02553b524ad74bbf,c90a1e1f5d6ba8aa
Normal-4576.png,Normal,Normal/images/Normal-4576.png,Normal/masks/Normal-4576.png,True,G13581,train,6df613d65105821a88f6d92e78245e3c,807f071e7a65e865
Normal-4577.png,Normal,Normal/images/Normal-4577.png,Normal/masks/Normal-4577.png,True,G13582,train,8674aafb8654228d3f7dc4786c0b997e,897f861a1d4e5b19
Normal-4578.png,Normal,Normal/images/Normal-4578.png,Normal/masks/Normal-4578.png,True,G13583,train,8989852891b4d5d6660b67247a8c38a6,d93cc61ad14f184f
Normal-4579.png,Normal,Normal/images/Normal-4579.png,Normal/masks/Normal-4579.png,True,G13584,train,5727b5fdfb57d891d46a46a0b6c87ba1,d92956165b5d0b1b
Normal-458.png,Normal,Normal/images/Normal-458.png,Normal/masks/Normal-458.png,True,G13585,train,0daf78669c7c715f41fb55e2eb4fd2e9,916ad65a59593d51
Normal-4580.png,Normal,Normal/images/Normal-4580.png,Normal/masks/Normal-4580.png,True,G13586,train,6786b49b6e14a6f42f9d76f74ee1a797,d13907061e5ff91a
Normal-4581.png,Normal,Normal/images/Normal-4581.png,Normal/masks/Normal-4581.png,True,G13587,val,d9dc8007aecc042508f18a97f57bfc33,cd06c34f446af4ea
Normal-4582.png,Normal,Normal/images/Normal-4582.png,Normal/masks/Normal-4582.png,True,G13588,val,83e09848da792846c70bf49cd488af48,dd39525f5a4a1a4a
Normal-4583.png,Normal,Normal/images/Normal-4583.png,Normal/masks/Normal-4583.png,True,G13589,val,dc5bca5ef5363a534bba64b87c001918,997b05161e474e5e
Normal-4584.png,Normal,Normal/images/Normal-4584.png,Normal/masks/Normal-4584.png,True,G13590,train,ae5433756ff57caac8d9a9cc52fac9e0,fd7dd0921b451c19
Normal-4585.png,Normal,Normal/images/Normal-4585.png,Normal/masks/Normal-4585.png,True,G13591,test,c7875faf9ca9b538daf33eba02dc5aa2,d13b861a19297f3a
Normal-4586.png,Normal,Normal/images/Normal-4586.png,Normal/masks/Normal-4586.png,True,G13592,test,60a52be6ce2773cc48b3e7c56fe92795,847ec2194e4f7c4e
Normal-4587.png,Normal,Normal/images/Normal-4587.png,Normal/masks/Normal-4587.png,True,G13593,train,994c0ae70e2d969e249accd141edafbf,9168861a1f4f396f
Normal-4588.png,Normal,Normal/images/Normal-4588.png,Normal/masks/Normal-4588.png,True,G13594,train,74ceada0631076dd927f03c1ba736753,b9798712c51f094f
Normal-4589.png,Normal,Normal/images/Normal-4589.png,Normal/masks/Normal-4589.png,True,G13595,train,09273f8c0825708f681b593e4360a82e,dd2ad30f0ca43ce5
Normal-459.png,Normal,Normal/images/Normal-459.png,Normal/masks/Normal-459.png,True,G13596,train,9c292310a024f65145483b639f5a8a03,c12ae21bbec94cdc
Normal-4590.png,Normal,Normal/images/Normal-4590.png,Normal/masks/Normal-4590.png,True,G13597,train,fc35ddd014f717fd2344e9f2b7c35b10,cc2f424b0e4c797d
Normal-4591.png,Normal,Normal/images/Normal-4591.png,Normal/masks/Normal-4591.png,True,G13598,train,a95cab4dc051d71125a3644bc559cca2,cd3cc7431b496e38
Normal-4592.png,Normal,Normal/images/Normal-4592.png,Normal/masks/Normal-4592.png,True,G13599,train,a1353d8f5880a2dc6dcc4042b4848b58,d82d4247195c6f3d
Normal-4593.png,Normal,Normal/images/Normal-4593.png,Normal/masks/Normal-4593.png,True,G13600,train,c2b2332711fe675c5cea75ba1f243e49,993d87121d4e1b9d
Normal-4594.png,Normal,Normal/images/Normal-4594.png,Normal/masks/Normal-4594.png,True,G13601,train,0418ffe54fa85fde8d3fb92c7f218f87,897a05161e9fe999
Normal-4595.png,Normal,Normal/images/Normal-4595.png,Normal/masks/Normal-4595.png,True,G13602,train,f2a371d425f652dcf027b41ad2ba8fb4,c83fc7161cc61f1c
Normal-4596.png,Normal,Normal/images/Normal-4596.png,Normal/masks/Normal-4596.png,True,G13603,train,c6e6c005a1ff5c81a4492bcab1b24964,d83d43031d1e2f3d
Normal-4597.png,Normal,Normal/images/Normal-4597.png,Normal/masks/Normal-4597.png,True,G13604,val,6440be4f5b5fd07c0a710fa9a9a177bb,ee2cc30b0f4e0f0f
Normal-4598.png,Normal,Normal/images/Normal-4598.png,Normal/masks/Normal-4598.png,True,G13605,val,4bea3ffa1defbb956c549a92da55f456,9979071615873e97
Normal-4599.png,Normal,Normal/images/Normal-4599.png,Normal/masks/Normal-4599.png,True,G13606,val,6c6b16a4cefa85e98b7f528fa8677971,8169865e5a9e5e39
Normal-46.png,Normal,Normal/images/Normal-46.png,Normal/masks/Normal-46.png,True,G13607,test,a6ddf04a961d3ac83532ce3b9b7a9e45,d12d061e5b6d7869
Normal-460.png,Normal,Normal/images/Normal-460.png,Normal/masks/Normal-460.png,True,G13608,test,d3d5fb66a0dd645afcd7ff905f0baefb,917fc71a8cc139d4
Normal-4600.png,Normal,Normal/images/Normal-4600.png,Normal/masks/Normal-4600.png,True,G13609,test,6921a1193f5b1660a0613974a9ee3a73,993d461b1f4e391a
Normal-4601.png,Normal,Normal/images/Normal-4601.png,Normal/masks/Normal-4601.png,True,G13610,train,5ecdd8915f4405869ce0d1ca8bd4f7ec,d13f461d180f393b
Normal-4602.png,Normal,Normal/images/Normal-4602.png,Normal/masks/Normal-4602.png,True,G13611,train,58996dd43f067a593e2909b49d0f2e7f,dd2ec25b424b1bc3
Normal-4603.png,Normal,Normal/images/Normal-4603.png,Normal/masks/Normal-4603.png,True,G13612,train,85e2f29ea2c8acf9d3b23db466b9160b,993d46431b786c79
Normal-4604.png,Normal,Normal/images/Normal-4604.png,Normal/masks/Normal-4604.png,True,G13613,train,2504e8acbb16946670085cf863f65622,9578064e792de4b6
Normal-4605.png,Normal,Normal/images/Normal-4605.png,Normal/masks/Normal-4605.png,True,G13614,train,94329e0f4fa4ee7243bbc95dda258e48,994ac61a092f3daf
Normal-4606.png,Normal,Normal/images/Normal-4606.png,Normal/masks/Normal-4606.png,True,G13615,val,d5bdade1f1038dbf962caf7ee6be4112,897fc61e144e1fc1
Normal-4607.png,Normal,Normal/images/Normal-4607.png,Normal/masks/Normal-4607.png,True,G13616,test,95400b353d4bfb87c91c2476d1effa37,c93e864b1e4d4b1e
Normal-4608.png,Normal,Normal/images/Normal-4608.png,Normal/masks/Normal-4608.png,True,G13617,train,aa5ec1157b1f51d9ce7457a3b05273d5,fd7d86d3194c4c28
Normal-4609.png,Normal,Normal/images/Normal-4609.png,Normal/masks/Normal-4609.png,True,G13618,train,9e81f7c0f4e761fc8230a6b9b71636aa,99a96c9e1959cd68
Normal-461.png,Normal,Normal/images/Normal-461.png,Normal/masks/Normal-461.png,True,G13619,train,5d5d8cbf3fc133af91f09bde7e026aee,986dc71acb4b384d
Normal-4610.png,Normal,Normal/images/Normal-4610.png,Normal/masks/Normal-4610.png,True,G13620,train,17f3fa1036cec2e2da7bdc9204145307,d0ec871b0bc63ad3
Normal-4611.png,Normal,Normal/images/Normal-4611.png,Normal/masks/Normal-4611.png,True,G13621,train,475e7b45cc0bf3eefe8550e0df51f083,d52e42574b4a4f4b
Normal-4612.png,Normal,Normal/images/Normal-4612.png,Normal/masks/Normal-4612.png,True,G13622,train,6a27e58911b7844c7e79a19c5aff17f7,9968c6151e5d7a6a
Normal-4613.png,Normal,Normal/images/Normal-4613.png,Normal/masks/Normal-4613.png,True,G13623,train,3eb2aae071264055d4a9a7c4cf7ce4f5,9929841e0d5f1b5f
Normal-4614.png,Normal,Normal/images/Normal-4614.png,Normal/masks/Normal-4614.png,True,G13624,val,cc67844167866be8f0679783eed03bc8,997d871ad94d3a11
Normal-4615.png,Normal,Normal/images/Normal-4615.png,Normal/masks/Normal-4615.png,True,G13625,train,e6c26763454a2745dab78c39d322f067,893f041a5969fbe1
Normal-4616.png,Normal,Normal/images/Normal-4616.png,Normal/masks/Normal-4616.png,True,G13626,train,59a8e19447ad1ea93233a1c944797a0b,9829c55617c71ed5
Normal-4617.png,Normal,Normal/images/Normal-4617.png,Normal/masks/Normal-4617.png,True,G13627,val,aa10c64dceb2f6aefc48cdef7ed1fda4,897dc612195f195d
Normal-4618.png,Normal,Normal/images/Normal-4618.png,Normal/masks/Normal-4618.png,True,G13628,test,788d1a051d1665c4c50788d6a66192d9,9939c61e1a0e3bb9
Normal-4619.png,Normal,Normal/images/Normal-4619.png,Normal/masks/Normal-4619.png,True,G13629,train,89abd54411497423d8e8c9c0e58c7306,dd0e434f4d0e4c6b
Normal-462.png,Normal,Normal/images/Normal-462.png,Normal/masks/Normal-462.png,True,G13630,val,2279f078d7bbbd5df041d8fd0dfc16ab,d03cc71ee92d1ac6
Normal-4620.png,Normal,Normal/images/Normal-4620.png,Normal/masks/Normal-4620.png,True,G13631,test,3c4fbbb1bd0381c3caf77df03b627f56,a97d161819db19cd
Normal-4621.png,Normal,Normal/images/Normal-4621.png,Normal/masks/Normal-4621.png,True,G13632,train,f76f7cfcce03291cd1d94426e72054b3,c139564a18bdedb4
Normal-4622.png,Normal,Normal/images/Normal-4622.png,Normal/masks/Normal-4622.png,True,G13633,train,b9d738f6c6c359c53e47c19c4acff62b,9929161a5b4d7af8
Normal-4623.png,Normal,Normal/images/Normal-4623.png,Normal/masks/Normal-4623.png,True,G13634,train,7557d812c61f12bdfcd589c4607305a8,c03d4b4f2dce3dc0
Normal-4624.png,Normal,Normal/images/Normal-4624.png,Normal/masks/Normal-4624.png,True,G13635,train,2194708b483dde39030c862862a24e3d,dc7ed25f40425b54
Normal-4625.png,Normal,Normal/images/Normal-4625.png,Normal/masks/Normal-4625.png,True,G13636,train,e379134d7f1af7a1d90485a704d90fc6,d97fc61e194e3841
Normal-4626.png,Normal,Normal/images/Normal-4626.png,Normal/masks/Normal-4626.png,True,G13637,train,692c9e8f97d92e01c004c8150125e97b,dc2d430b4e2f0e2f
Normal-4627.png,Normal,Normal/images/Normal-4627.png,Normal/masks/Normal-4627.png,True,G13638,train,5f443403bfe9125189b78296f3d429cf,c0bc7ecb3c309372
Normal-4628.png,Normal,Normal/images/Normal-4628.png,Normal/masks/Normal-4628.png,True,G13639,train,d1eaafb0b5da8fd09434da59fbfcca5e,d839d61ad44b3d4a
Normal-4629.png,Normal,Normal/images/Normal-4629.png,Normal/masks/Normal-4629.png,True,G13640,train,01adb2a7644e10235d2e49181eab5968,d528065f0b5e687d
Normal-463.png,Normal,Normal/images/Normal-463.png,Normal/masks/Normal-463.png,True,G13641,train,8e1e8c2222ac92071cc543f6fc02ed84,d03dc71abc6c3cc4
Normal-4630.png,Normal,Normal/images/Normal-4630.png,Normal/masks/Normal-4630.png,True,G13642,train,cc942159c8e1cb58d407a6dac446498b,c93cc71b164f1a69
Normal-4631.png,Normal,Normal/images/Normal-4631.png,Normal/masks/Normal-4631.png,True,G13643,train,1d5c5a275abac65c198f4d7012e0fe66,c938461dc8ebb9e4
Normal-4632.png,Normal,Normal/images/Normal-4632.png,Normal/masks/Normal-4632.png,True,G13644,train,1577ae0f181a8c072050ef1028e5ff2a,d43fc3435d4c487c
Normal-4633.png,Normal,Normal/images/Normal-4633.png,Normal/masks/Normal-4633.png,True,G13645,val,9090f9696d0d874d224223361fc1194c,d42ed2c34b5d4647
Normal-4634.png,Normal,Normal/images/Normal-4634.png,Normal/masks/Normal-4634.png,True,G13646,val,f5a2f15c782bbde79d2973006a534a03,d92f461dc61fd02c
Normal-4635.png,Normal,Normal/images/Normal-4635.png,Normal/masks/Normal-4635.png,True,G13647,val,1f2c9a1baf94e01b99c3650ede38467e,d139d616184f5f1a
Normal-4636.png,Normal,Normal/images/Normal-4636.png,Normal/masks/Normal-4636.png,True,G13648,test,fd172352a8c69c4a28717425e43a2ac4,842f424b0f4e7f3c
Normal-4637.png,Normal,Normal/images/Normal-4637.png,Normal/masks/Normal-4637.png,True,G13649,test,30c83c7736745737208446160d4e2554,897f061a186a3b7d
Normal-4638.png,Normal,Normal/images/Normal-4638.png,Normal/masks/Normal-4638.png,True,G13650,test,c328f569da5bbf3adb464630a35ac70e,987f861e485b78e1
Normal-4639.png,Normal,Normal/images/Normal-4639.png,Normal/masks/Normal-4639.png,True,G13651,train,7f834ca5b684ea71805ad64e690243c7,9c6ec35b0d4d0e4e
Normal-464.png,Normal,Normal/images/Normal-464.png,Normal/masks/Normal-464.png,True,G13652,train,b79badd3ca169af06585edde1c4b48eb,d06c8739e92c69ce
Normal-4640.png,Normal,Normal/images/Normal-4640.png,Normal/masks/Normal-4640.png,True,G13653,train,ff4888cc57644c11862e5904669296f5,dc2d420b0fce0f3d
Normal-4641.png,Normal,Normal/images/Normal-4641.png,Normal/masks/Normal-4641.png,True,G13654,train,c6a36fc8ae4625f96da163bb37361867,dd2e025b592c6e78
Normal-4642.png,Normal,Normal/images/Normal-4642.png,Normal/masks/Normal-4642.png,True,G13655,train,b91dd56feb90ef67e615b117b77e350a,d83d421b0fca0f1f
Normal-4643.png,Normal,Normal/images/Normal-4643.png,Normal/masks/Normal-4643.png,True,G13656,train,9b960f1180e3fd29860c6b498921e3cb,dc2c424f0f4ebda8
Normal-4644.png,Normal,Normal/images/Normal-4644.png,Normal/masks/Normal-4644.png,True,G13657,train,cb8aa158b937aa913383b4f14aad232b,dd0e47594deea281
Normal-4645.png,Normal,Normal/images/Normal-4645.png,Normal/masks/Normal-4645.png,True,G13658,train,a25797050b9cdb14be2350c682970e4b,9969871a085f7b4d
Normal-4646.png,Normal,Normal/images/Normal-4646.png,Normal/masks/Normal-4646.png,True,G13659,train,34d795b783183dd29e444d3873f19ed3,952f06590f4e786d
Normal-4647.png,Normal,Normal/images/Normal-4647.png,Normal/masks/Normal-4647.png,True,G13660,test,8f958e834a73bafe4282d96473961139,807b07961e135f5b
Normal-4648.png,Normal,Normal/images/Normal-4648.png,Normal/masks/Normal-4648.png,True,G13661,train,e10586b1c28469d96392239223a3b581,dd2ed45b0c6c0e2d
Normal-4649.png,Normal,Normal/images/Normal-4649.png,Normal/masks/Normal-4649.png,True,G13662,train,8d66d06eca0824204c345e2628c06c01,d929c6171e4a1b3d
Normal-465.png,Normal,Normal/images/Normal-465.png,Normal/masks/Normal-465.png,True,G13663,train,41df25ca7db4c9feb5be099045c666dd,917be63a4b193945
Normal-4650.png,Normal,Normal/images/Normal-4650.png,Normal/masks/Normal-4650.png,True,G13664,train,1524a361efd4e65591aa4211754ec79b,d93bc2071e841f3d
Normal-4651.png,Normal,Normal/images/Normal-4651.png,Normal/masks/Normal-4651.png,True,G13665,train,72e9bc8c2dff9bbffcef7de5a6cb987e,8d2d921bc50e5d4f
Normal-4652.png,Normal,Normal/images/Normal-4652.png,Normal/masks/Normal-4652.png,True,G13666,val,1f6d9fee251ed2cf6d2095206e791a6d,dd2c4612434f1f5b
Normal-4653.png,Normal,Normal/images/Normal-4653.png,Normal/masks/Normal-4653.png,True,G13667,val,115e3ee37d2eac1b725b9e2baef50151,dd2cc2535dd2421f
Normal-4654.png,Normal,Normal/images/Normal-4654.png,Normal/masks/Normal-4654.png,True,G13668,val,e2b502a3475ba6df0d34ecbd0ff10479,dc3d420bc44ebde4
Normal-4655.png,Normal,Normal/images/Normal-4655.png,Normal/masks/Normal-4655.png,True,G13669,test,b2feeba200f281f79236c9224e499d3d,d03d42555d5a4b5d
Normal-4656.png,Normal,Normal/images/Normal-4656.png,Normal/masks/Normal-4656.png,True,G13670,test,63d9b78d272b13945de251fedb947037,993f061a4d7a39e4
Normal-4657.png,Normal,Normal/images/Normal-4657.png,Normal/masks/Normal-4657.png,True,G13671,train,5b9cb496377f46e2d6bb1635583ca499,dc2a530f4e2e0e2f
Normal-4658.png,Normal,Normal/images/Normal-4658.png,Normal/masks/Normal-4658.png,True,G13672,train,75721d3a7403f603056c78d6ca13f92b,991dc75e494f1c58
Normal-4659.png,Normal,Normal/images/Normal-4659.png,Normal/masks/Normal-4659.png,True,G13673,train,b5095f1144664183758765181730390a,d93dd2471f14491b
Normal-466.png,Normal,Normal/images/Normal-466.png,Normal/masks/Normal-466.png,True,G13674,train,f723d075968e4520d24524d3b1141252,917da71a890d3d65
Normal-4660.png,Normal,Normal/images/Normal-4660.png,Normal/masks/Normal-4660.png,True,G13675,train,24f4d03a722ae281f4e556ddac8c92b8,8179c71a58577a1b
Normal-4661.png,Normal,Normal/images/Normal-4661.png,Normal/masks/Normal-4661.png,True,G13676,train,e039ebdc339a2b0afa6ac8544a9614e1,cd2c96531d4f4a1d
Normal-4662.png,Normal,Normal/images/Normal-4662.png,Normal/masks/Normal-4662.png,True,G13677,val,029ee926ebf4c78382d72c15bf3c6e3b,916b461a1bef3ab0
Normal-4663.png,Normal,Normal/images/Normal-4663.png,Normal/masks/Normal-4663.png,True,G13678,train,9846a874b64809a9c557c04a63d776e7,d939d5421f85461f
Normal-4664.png,Normal,Normal/images/Normal-4664.png,Normal/masks/Normal-4664.png,True,G13679,val,56561b58d0a62731f1617c6a89d0e489,993d869f1d4f1838
Normal-4665.png,Normal,Normal/images/Normal-4665.png,Normal/masks/Normal-4665.png,True,G13680,train,5ad2fcb45179c410efd577070f75c524,dd3d861a514f1b4c
Normal-4666.png,Normal,Normal/images/Normal-4666.png,Normal/masks/Normal-4666.png,True,G13681,test,d63a8246a7ff54f8e961f6a0ccbda95b,997f141a5f0838bd
Normal-4667.png,Normal,Normal/images/Normal-4667.png,Normal/masks/Normal-4667.png,True,G13682,train,16f8ece9dfa59b05d353e53f1b433add,d81bd21c1b1b1e2f
Normal-4668.png,Normal,Normal/images/Normal-4668.png,Normal/masks/Normal-4668.png,True,G13683,train,d440620b8789ed25dcb06202f026cd41,887d45561e5b4d1e
Normal-4669.png,Normal,Normal/images/Normal-4669.png,Normal/masks/Normal-4669.png,True,G13684,train,3c9fd9a1464c4f0ccab1e4de99c57f9e,dc3cd25b0cca98b9
Normal-467.png,Normal,Normal/images/Normal-467.png,Normal/masks/Normal-467.png,True,G13685,train,df4c03666ee6284aa3393a5b55ac80a4,9169c61af9693dc1
Normal-4670.png,Normal,Normal/images/Normal-4670.png,Normal/masks/Normal-4670.png,True,G13686,train,6032e37d3a01c788c8de1b4a1774a7b9,9979861e9e157a58
Normal-4671.png,Normal,Normal/images/Normal-4671.png,Normal/masks/Normal-4671.png,True,G13687,train,c989d0c40626f571b411bd50dbe19833,d93bd61a554a5d42
Normal-4672.png,Normal,Normal/images/Normal-4672.png,Normal/masks/Normal-4672.png,True,G13688,val,0d1a3ad6fcf52b23927ef390cc1e1b15,d93dc2431fc50b1d
Normal-4673.png,Normal,Normal/images/Normal-4673.png,Normal/masks/Normal-4673.png,True,G13689,train,203643783e51fb9c8022ff89ea99a4c7,897d861a1d5b194f
Normal-4674.png,Normal,Normal/images/Normal-4674.png,Normal/masks/Normal-4674.png,True,G13690,test,dea8ef243e0304bece3671b94820ca54,8c4e434f4d2eaeac
Normal-4675.png,Normal,Normal/images/Normal-4675.png,Normal/masks/Normal-4675.png,True,G13691,train,f871d30dcbda56a299afa04f82267665,dd3cd30f8a0b0f8c
Normal-4676.png,Normal,Normal/images/Normal-4676.png,Normal/masks/Normal-4676.png,True,G13692,test,9babf1f646cdea4becc5c5859f7bba3b,dd2ec20b494c3f2d
Normal-4677.png,Normal,Normal/images/Normal-4677.png,Normal/masks/Normal-4677.png,True,G13693,train,baca847f11288776b2bca5268c770727,917f470a1a4b3b39
Normal-4678.png,Normal,Normal/images/Normal-4678.png,Normal/masks/Normal-4678.png,True,G13694,train,b70ba179cbd76404c243c417c39fa877,f969861ec452595e
Normal-4679.png,Normal,Normal/images/Normal-4679.png,Normal/masks/Normal-4679.png,True,G13695,train,8dd2f85641f59091e3646cbecc3b2ff8,996ec7190f0f386c
Normal-468.png,Normal,Normal/images/Normal-468.png,Normal/masks/Normal-468.png,True,G13696,train,8b1661ec768a8683177117002c375aed,986bc21adf591d49
Normal-4680.png,Normal,Normal/images/Normal-4680.png,Normal/masks/Normal-4680.png,True,G13697,train,bcdaef3a05fa3ef37f5a4f770ad2860e,d94b4f1f180f0f1c
Normal-4681.png,Normal,Normal/images/Normal-4681.png,Normal/masks/Normal-4681.png,True,G13698,train,24abb33b2bdf0d10d797b39ac724a7a3,dd2cc21f8c0e1f9c
Normal-4682.png,Normal,Normal/images/Normal-4682.png,Normal/masks/Normal-4682.png,True,G13699,train,05280dd3fe63a4a2d90e431291815ff9,983905161e5be97e
Normal-4683.png,Normal,Normal/images/Normal-4683.png,Normal/masks/Normal-4683.png,True,G13700,train,38a7aa89f94b97ea40bb832383f35f1d,993b871f144f195a
Normal-4684.png,Normal,Normal/images/Normal-4684.png,Normal/masks/Normal-4684.png,True,G13701,train,adcf5ba3cb1fe20cdf7b574d88fd066c,95e89f83f1b4460b
Normal-4685.png,Normal,Normal/images/Normal-4685.png,Normal/masks/Normal-4685.png,True,G13702,train,1880b4c908828e4793e7b66603ff51b5,897f061e194b1f5a
Normal-4686.png,Normal,Normal/images/Normal-4686.png,Normal/masks/Normal-4686.png,True,G13703,train,485be8b8baa9d61e9cda73694599b6e6,8d2e524b4b456b5d
Normal-4687.png,Normal,Normal/images/Normal-4687.png,Normal/masks/Normal-4687.png,True,G13704,train,662f20dbea62f1a3066f671a809d5cde,897bc71a095a39cd
Normal-4688.png,Normal,Normal/images/Normal-4688.png,Normal/masks/Normal-4688.png,True,G13705,train,a265b27e69cf61bc3be9e2b1d71ea922,dd3dc25b1b450f18
Normal-4689.png,Normal,Normal/images/Normal-4689.png,Normal/masks/Normal-4689.png,True,G13706,test,caba93e62c5534e07506f419bc1a3622,8978871619d75e59
Normal-469.png,Normal,Normal/images/Normal-469.png,Normal/masks/Normal-469.png,True,G13707,test,07413307da02ca087a5ee9dfa16535fa,d0fec71ab98d28a1
Normal-4690.png,Normal,Normal/images/Normal-4690.png,Normal/masks/Normal-4690.png,True,G13708,val,2c68692cc60c2a4be3cf6b815f30dada,dc3dc21b0a0e3dbc
Normal-4691.png,Normal,Normal/images/Normal-4691.png,Normal/masks/Normal-4691.png,True,G13709,train,dc01da5ce2e30b052f57be1c238cb477,d93bc6161f4c5a38
Normal-4692.png,Normal,Normal/images/Normal-4692.png,Normal/masks/Normal-4692.png,True,G13710,test,20773f9cf5c35fac751ff9b212e78256,c92d061edc7a78e1
Normal-4693.png,Normal,Normal/images/Normal-4693.png,Normal/masks/Normal-4693.png,True,G13711,train,51042b0a3caade004d9cbf590ceb7128,d95917161a19593f
Normal-4694.png,Normal,Normal/images/Normal-4694.png,Normal/masks/Normal-4694.png,True,G13712,val,431d578074d95c79f2798e487a293a9c,dd7cc20b8b0f388b
Normal-4695.png,Normal,Normal/images/Normal-4695.png,Normal/masks/Normal-4695.png,True,G13713,val,4fa770d3d9339179a8503f3aac51bc72,d43d43530f0e2f1d
Normal-4696.png,Normal,Normal/images/Normal-4696.png,Normal/masks/Normal-4696.png,True,G13714,train,dd465085baa71d79cbb5376a6243d868,dd3d42431f492c3d
Normal-4697.png,Normal,Normal/images/Normal-4697.png,Normal/masks/Normal-4697.png,True,G13715,train,563b080a30f993797ddc28d6329d1f91,8825c21f0b1f3daf
Normal-4698.png,Normal,Normal/images/Normal-4698.png,Normal/masks/Normal-4698.png,True,G13716,train,76de0c20c8648f047480f34e4a588b1f,997b47961b494f18
Normal-4699.png,Normal,Normal/images/Normal-4699.png,Normal/masks/Normal-4699.png,True,G13717,train,523fccd2c04ccc21cbf80d1f17d4201d,d84d87561e4d191f
Normal-47.png,Normal,Normal/images/Normal-47.png,Normal/masks/Normal-47.png,True,G13718,train,3d2a50abb88d627ce92789648e7a40a6,997bd718045b3b8a
Normal-470.png,Normal,Normal/images/Normal-470.png,Normal/masks/Normal-470.png,True,G13719,train,7229b31cdf4935ff515905e6a7698049,907dc71a9d4d3c94
Normal-4700.png,Normal,Normal/images/Normal-4700.png,Normal/masks/Normal-4700.png,True,G13720,train,8e435149557cd649176c34a2b7a6990a,b97d871a8716105f
Normal-4701.png,Normal,Normal/images/Normal-4701.png,Normal/masks/Normal-4701.png,True,G13721,train,9cd701f2fca51b6a847270e266b9ad0f,817bd61859437b59
Normal-4702.png,Normal,Normal/images/Normal-4702.png,Normal/masks/Normal-4702.png,True,G13722,train,af3ac6788e74c20fdddf013094869887,dd2ad65b1d440b1d
Normal-4703.png,Normal,Normal/images/Normal-4703.png,Normal/masks/Normal-4703.png,True,G13723,train,091252a6aefb9b2321c8a85d66adbffc,d42d82531f0f1f1d
Normal-4704.png,Normal,Normal/images/Normal-4704.png,Normal/masks/Normal-4704.png,True,G13724,train,295127a77ac9d50986e1d881e055ec96,813956461b786beb
Normal-4705.png,Normal,Normal/images/Normal-4705.png,Normal/masks/Normal-4705.png,True,G13725,train,41d9b02ca31fad4fdbd69b664df8a9bc,d85f57421f4d1d09
Normal-4706.png,Normal,Normal/images/Normal-4706.png,Normal/masks/Normal-4706.png,True,G13726,train,50e52b9a290b380a9c7113a22aaaece9,d42e42438b5b4d6f
Normal-4707.png,Normal,Normal/images/Normal-4707.png,Normal/masks/Normal-4707.png,True,G13727,val,d617b6d1879254037533ec4728e490aa,dd2f460a08ade93d
Normal-4708.png,Normal,Normal/images/Normal-4708.png,Normal/masks/Normal-4708.png,True,G13728,val,dab32cd88268346238a3c6ccd0634e26,9d7c861a5b4c4e78
Normal-4709.png,Normal,Normal/images/Normal-4709.png,Normal/masks/Normal-4709.png,True,G13729,test,a70b26664ea67bce0bf7f8ac1d7c7788,993987061e4f4f1e
Normal-471.png,Normal,Normal/images/Normal-471.png,Normal/masks/Normal-471.png,True,G13730,val,fe61662d658239df1b6e63769cbfd6c2,d53e873a0d0d2cce
Normal-4710.png,Normal,Normal/images/Normal-4710.png,Normal/masks/Normal-4710.png,True,G13731,test,8808abcb4e2de352b8e20637d4a6248d,817d16825f977a98
Normal-4711.png,Normal,Normal/images/Normal-4711.png,Normal/masks/Normal-4711.png,True,G13732,test,758cbf081b1b24a7109a14c40a35e492,913bc95a397966a4
Normal-4712.png,Normal,Normal/images/Normal-4712.png,Normal/masks/Normal-4712.png,True,G13733,train,254a18d9d2e3f5766ed993a5c8dc7812,dd2cd0834fc5074f
Normal-4713.png,Normal,Normal/images/Normal-4713.png,Normal/masks/Normal-4713.png,True,G13734,train,4c8744fa6325ae9174ed85f8b66f527c,dd2a161f081f5d1d
Normal-4714.png,Normal,Normal/images/Normal-4714.png,Normal/masks/Normal-4714.png,True,G13735,train,6dccb91747ee22cc609dbac7c53537de,d42dc3c30a4f435f
Normal-4715.png,Normal,Normal/images/Normal-4715.png,Normal/masks/Normal-4715.png,True,G13736,val,aa9afc238aa01da50c5a1bf308d50be0,997dc60a1b4c6a3d
Normal-4716.png,Normal,Normal/images/Normal-4716.png,Normal/masks/Normal-4716.png,True,G13737,val,328938347862823d65a9a190690bbef0,99698616d34fb84b
Normal-4717.png,Normal,Normal/images/Normal-4717.png,Normal/masks/Normal-4717.png,True,G13738,train,2f3c0f3dc715a4a26c3ff0775c998f6d,9969061a4d1e5d5f
Normal-4718.png,Normal,Normal/images/Normal-4718.png,Normal/masks/Normal-4718.png,True,G13739,train,052c41e8eb08a0171118cec1cb6fdb0d,dd2ed04b1f4d0c3c
Normal-4719.png,Normal,Normal/images/Normal-4719.png,Normal/masks/Normal-4719.png,True,G13740,train,a158d73490255f41f3260eeb03d82c6f,997d861e495d5945
Normal-472.png,Normal,Normal/images/Normal-472.png,Normal/masks/Normal-472.png,True,G13741,train,0bc6959ba53e608045e08c1a800a794f,915ec61acda53c96
Normal-4720.png,Normal,Normal/images/Normal-4720.png,Normal/masks/Normal-4720.png,True,G13742,train,aa02fdabf315d3bb0e1a361a72e5eb5d,d93fc6431f0d4919
Normal-4721.png,Normal,Normal/images/Normal-4721.png,Normal/masks/Normal-4721.png,True,G13743,train,c9cbb42e5162984678bab43b51fc084a,d42ec34b4d4e0f4e
Normal-4722.png,Normal,Normal/images/Normal-4722.png,Normal/masks/Normal-4722.png,True,G13744,train,71f5ed2c2f89da85403a2d92e856c67f,d42f435f080f0f3e
Normal-4723.png,Normal,Normal/images/Normal-4723.png,Normal/masks/Normal-4723.png,True,G13745,train,bbca6da7859f6a8af88693eb20294a4b,dc3d42434f4e4e4e
Normal-4724.png,Normal,Normal/images/Normal-4724.png,Normal/masks/Normal-4724.png,True,G13746,train,435e70af18b195873580da42093d836e,d42ad3535d5c705a
Normal-4725.png,Normal,Normal/images/Normal-4725.png,Normal/masks/Normal-4725.png,True,G13747,train,8090c383983ed5a5493214b6e8bc4426,dd2cd21373431b53
Normal-4726.png,Normal,Normal/images/Normal-4726.png,Normal/masks/Normal-4726.png,True,G13748,train,f5084693c06643afcbade7ce3af0c8c8,993c46475a1bd64b
Normal-4727.png,Normal,Normal/images/Normal-4727.png,Normal/masks/Normal-4727.png,True,G13749,train,a3819bf109e05ce4c40a7f39d3232318,c93d06431f1e6c6d
Normal-4728.png,Normal,Normal/images/Normal-4728.png,Normal/masks/Normal-4728.png,True,G13750,val,a758bdf6b1b06811757571548f8890e0,dd4a430f0e3d3437
Normal-4729.png,Normal,Normal/images/Normal-4729.png,Normal/masks/Normal-4729.png,True,G13751,train,5f685280eacfd4103ab265c0dcf3154a,996c161a4d5879cf
Normal-473.png,Normal,Normal/images/Normal-473.png,Normal/masks/Normal-473.png,True,G13752,test,a5ebd0c594e99e1425eaab175fa2c142,986f871a831b4d5d
Normal-4730.png,Normal,Normal/images/Normal-4730.png,Normal/masks/Normal-4730.png,True,G13753,test,01e7a6b7ae43d52882c6b68f69e7a4ae,d83d821d1b4f2db8
Normal-4731.png,Normal,Normal/images/Normal-4731.png,Normal/masks/Normal-4731.png,True,G13754,test,8bb16d3e84d084b4a40dc007b400765e,997fc61a1d5a1945
Normal-4732.png,Normal,Normal/images/Normal-4732.png,Normal/masks/Normal-4732.png,True,G13755,train,b6442ac2060231140910a984608f8a26,8969c7121d4f5b1d
Normal-4733.png,Normal,Normal/images/Normal-4733.png,Normal/masks/Normal-4733.png,True,G13756,train,c60ddf73b48efa866f406e2bdd743dc1,8978565b5ed738c0
Normal-4734.png,Normal,Normal/images/Normal-4734.png,Normal/masks/Normal-4734.png,True,G13757,train,916bfb97ce1b5b327d0b8d0e7d10ba4a,91681e7a5b797360
Normal-4735.png,Normal,Normal/images/Normal-4735.png,Normal/masks/Normal-4735.png,True,G13758,train,a822abb800a34e0b95db0f5f1dd2f4d0,dc2e43d12c2cbf38
Normal-4736.png,Normal,Normal/images/Normal-4736.png,Normal/masks/Normal-4736.png,True,G13759,train,0896c4f1db790198322832909b31f03e,997f871a1d0a19ab
Normal-4737.png,Normal,Normal/images/Normal-4737.png,Normal/masks/Normal-4737.png,True,G13760,train,921aae1627840a18ddad14b3d19b8e24,897b041417de3b97
Normal-4738.png,Normal,Normal/images/Normal-4738.png,Normal/masks/Normal-4738.png,True,G13761,train,e465d754874ba0eebbfe169e49eafb40,c93d82171a5f4e1b
Normal-4739.png,Normal,Normal/images/Normal-4739.png,Normal/masks/Normal-4739.png,True,G13762,train,d4da1ee7a82204d979a0c93c6437f077,997d471e145e1c4e
Normal-474.png,Normal,Normal/images/Normal-474.png,Normal/masks/Normal-474.png,True,G13763,train,d884fd732c0458061b77a4ffb3d07a78,982fc61a5f5d1d44
Normal-4740.png,Normal,Normal/images/Normal-4740.png,Normal/masks/Normal-4740.png,True,G13764,val,3ad97d45c6d7e46faaeeae1c426fe88e,cc0d5743135f4d1b
Normal-4741.png,Normal,Normal/images/Normal-4741.png,Normal/masks/Normal-4741.png,True,G13765,test,cc9ce5607db64262bfd2b489e92f98aa,996c871a0f5f185d
Normal-4742.png,Normal,Normal/images/Normal-4742.png,Normal/masks/Normal-4742.png,True,G13766,train,beb0a286b6459815ac3964301f8d2fdd,d43d8342b90aef0f
Normal-4743.png,Normal,Normal/images/Normal-4743.png,Normal/masks/Normal-4743.png,True,G13767,train,4daf85558c9321b9f2503d6c6cf5f895,9d7c42c30f84bc79
Normal-4744.png,Normal,Normal/images/Normal-4744.png,Normal/masks/Normal-4744.png,True,G13768,train,71ad639a9c89a925913ba9f4187f9b97,dc2cc35b494e4e4d
Normal-4745.png,Normal,Normal/images/Normal-4745.png,Normal/masks/Normal-4745.png,True,G13769,val,0b45a3daf2c69e597cbf5df8b3a3b7f6,9d61c6da58d25a78
Normal-4746.png,Normal,Normal/images/Normal-4746.png,Normal/masks/Normal-4746.png,True,G13770,test,977641e7dbf6edd6cb9e229b0e58cf94,d17bd61618575a1a
Normal-4747.png,Normal,Normal/images/Normal-4747.png,Normal/masks/Normal-4747.png,True,G13771,val,1ebb11309fa29e19ce19fb42de8a8175,887dd619590afd0b
Normal-4748.png,Normal,Normal/images/Normal-4748.png,Normal/masks/Normal-4748.png,True,G13772,test,b996e79f525f816cfcdc3c17bd95b0a0,f94dc61e494928ed
Normal-4749.png,Normal,Normal/images/Normal-4749.png,Normal/masks/Normal-4749.png,True,G13773,train,dde3ef0db4278b4852a04191e2eeee63,cc5b5b2c9d3c9470
Normal-475.png,Normal,Normal/images/Normal-475.png,Normal/masks/Normal-475.png,True,G13774,train,c82eaef829bb5c214a5c2a796c48c1b3,c0a9c20bcb6b796d
Normal-4750.png,Normal,Normal/images/Normal-4750.png,Normal/masks/Normal-4750.png,True,G13775,train,4b458f80575f3f99b35cbf80afd1a474,d97b465e184b1b19
Normal-4751.png,Normal,Normal/images/Normal-4751.png,Normal/masks/Normal-4751.png,True,G13776,train,cb17212e33635d9b81843b1d9482276b,d52a5643531f4e4b
Normal-4752.png,Normal,Normal/images/Normal-4752.png,Normal/masks/Normal-4752.png,True,G13777,train,ba5a70fa80e4e027e590c5b5c5a9b4d2,d87bc50612565f1e
Normal-4753.png,Normal,Normal/images/Normal-4753.png,Normal/masks/Normal-4753.png,True,G13778,train,f062c8875b1b2c1d05f5c0d4e5c92180,f97f86d61a014b19
Normal-4754.png,Normal,Normal/images/Normal-4754.png,Normal/masks/Normal-4754.png,True,G13779,train,df86334a1373b4d8ab669e75c32bdbb2,91798687181f473f
Normal-4755.png,Normal,Normal/images/Normal-4755.png,Normal/masks/Normal-4755.png,True,G13780,train,669ef3b3517590ac1bdb48d3b4cacba7,c97b869616471b59
Normal-4756.png,Normal,Normal/images/Normal-4756.png,Normal/masks/Normal-4756.png,True,G13781,train,d4c82d0054de70987cc10b43f3c1d347,9178441e1a5b5f3b
Normal-4757.png,Normal,Normal/images/Normal-4757.png,Normal/masks/Normal-4757.png,True,G13782,train,e98804a0d6e558347af0351b620d0c3d,817b87121a4dbbba
Normal-4758.png,Normal,Normal/images/Normal-4758.png,Normal/masks/Normal-4758.png,True,G13783,train,19276f49bd342390274565675b4a59ff,817906561a17df3b
Normal-4759.png,Normal,Normal/images/Normal-4759.png,Normal/masks/Normal-4759.png,True,G13784,train,b0af82dc605afbce77177d3dd15541a2,9901061e195f5f5f
Normal-476.png,Normal,Normal/images/Normal-476.png,Normal/masks/Normal-476.png,True,G13785,train,5442fdb66e28d9a2caaa641884bfda67,946ea73bcf2c4846
Normal-4760.png,Normal,Normal/images/Normal-4760.png,Normal/masks/Normal-4760.png,True,G13786,train,24739f7db4151c02d8cd3b4f581ae1d4,c879d65f54491d4a
Normal-4761.png,Normal,Normal/images/Normal-4761.png,Normal/masks/Normal-4761.png,True,G13787,val,ff879d669ec58bdaac2539107217ae60,897f061a075f1b0f
Normal-4762.png,Normal,Normal/images/Normal-4762.png,Normal/masks/Normal-4762.png,True,G13788,val,ebdff91bcca96b222ddc09a0be9ca85a,bc65c31a4b4b1b4b
Normal-4763.png,Normal,Normal/images/Normal-4763.png,Normal/masks/Normal-4763.png,True,G13789,val,9f2ffa51454f3200b9b128798a3af7de,b939c6961acd1b1a
Normal-4764.png,Normal,Normal/images/Normal-4764.png,Normal/masks/Normal-4764.png,True,G13790,train,5ef45870989a6eba78184b34adadb45a,8978061f1c5f1b5e
Normal-4765.png,Normal,Normal/images/Normal-4765.png,Normal/masks/Normal-4765.png,True,G13791,test,cc1d17cb11ef514027ba7fc16d0964d8,9179c31e5a2d7929
Normal-4766.png,Normal,Normal/images/Normal-4766.png,Normal/masks/Normal-4766.png,True,G13792,test,8fa9a88950220cfa0031792ae7ef4256,d928531f461b4d5d
Normal-4767.png,Normal,Normal/images/Normal-4767.png,Normal/masks/Normal-4767.png,True,G13793,test,34e9c6039ae1ba57e3792ace352aedf5,b87947161e4b4f1c
Normal-4768.png,Normal,Normal/images/Normal-4768.png,Normal/masks/Normal-4768.png,True,G13794,train,ab4f43b44acf82d90708fe273aafa27d,993c060a5bee68f9
Normal-4769.png,Normal,Normal/images/Normal-4769.png,Normal/masks/Normal-4769.png,True,G13795,train,1505a76e2df536c27c5bb727b1813ae4,b82d871e971e3a4c
Normal-477.png,Normal,Normal/images/Normal-477.png,Normal/masks/Normal-477.png,True,G13796,train,5c64d09e774ee3953183035bf3a22e9f,913bc61a18cd5b9e
Normal-4770.png,Normal,Normal/images/Normal-4770.png,Normal/masks/Normal-4770.png,True,G13797,train,1da97fac93aa505af61cfd141a785e7a,dc2cd24b4fc45a1d
Normal-4771.png,Normal,Normal/images/Normal-4771.png,Normal/masks/Normal-4771.png,True,G13798,train,6d641d0d109e43595bbf9f40723e551f,dc2c03531d4c4ffc
Normal-4772.png,Normal,Normal/images/Normal-4772.png,Normal/masks/Normal-4772.png,True,G13799,train,e471b7ae6f46413de69920bcca70c16f,c9298616be8af9f0
Normal-4773.png,Normal,Normal/images/Normal-4773.png,Normal/masks/Normal-4773.png,True,G13800,train,25518f85ad2c27f39e70fed42ad51afa,dc3dc2431b8e2b2d
Normal-4774.png,Normal,Normal/images/Normal-4774.png,Normal/masks/Normal-4774.png,True,G13801,train,af5f63707dec91df6a3180a2e0827ecd,d5bf425f1d441a1c
Normal-4775.png,Normal,Normal/images/Normal-4775.png,Normal/masks/Normal-4775.png,True,G13802,train,eb3d0fedbe8a9482af8599ad4617683a,d93d06061b4e4e7b
Normal-4776.png,Normal,Normal/images/Normal-4776.png,Normal/masks/Normal-4776.png,True,G13803,train,ce87dc0f28536c0993ab5d42f34e3d1a,81f9c71a415c7a5d
Normal-4777.png,Normal,Normal/images/Normal-4777.png,Normal/masks/Normal-4777.png,True,G13804,train,3d008d3990a618304f1c66c152cf5aed,d93cc6531b070f1d
Normal-4778.png,Normal,Normal/images/Normal-4778.png,Normal/masks/Normal-4778.png,True,G13805,train,31bdccbc6227d8c2c75b9df08f8ebcde,d02f4b434929efe1
Normal-4779.png,Normal,Normal/images/Normal-4779.png,Normal/masks/Normal-4779.png,True,G13806,train,4f77e19044a2d8a234ff07b42611421d,d42fc34b094f2d6c
Normal-478.png,Normal,Normal/images/Normal-478.png,Normal/masks/Normal-478.png,True,G13807,val,8af000d53df643712ea96bde10e20302,997d870a870b1c6f
Normal-4780.png,Normal,Normal/images/Normal-4780.png,Normal/masks/Normal-4780.png,True,G13808,val,77be8baaa451baeb5f13950b5a0950be,c02dc21b154f3e7e
Normal-4781.png,Normal,Normal/images/Normal-4781.png,Normal/masks/Normal-4781.png,True,G13809,val,a516bebdd000522d1e831769268f2b38,dc28420f4a4f3fec
Normal-4782.png,Normal,Normal/images/Normal-4782.png,Normal/masks/Normal-4782.png,True,G13810,test,b127b17e0e79c3568addc24efdce75ae,987d871a0d4f1aad
Normal-4783.png,Normal,Normal/images/Normal-4783.png,Normal/masks/Normal-4783.png,True,G13811,test,57a34dbaba6de104a9c5a96147a6cd02,d42d820f1cecbdb8
Normal-4784.png,Normal,Normal/images/Normal-4784.png,Normal/masks/Normal-4784.png,True,G13812,test,15790401f01ac6847b43f61e3cfa5209,fc0c431b4cec3cad
Normal-4785.png,Normal,Normal/images/Normal-4785.png,Normal/masks/Normal-4785.png,True,G13813,train,4612fc2f5052c9bc065560d07f3ff4e4,d03d42c75f407d5a
Normal-4786.png,Normal,Normal/images/Normal-4786.png,Normal/masks/Normal-4786.png,True,G13814,train,10080280b9e81d14a1971876f9e1c6bc,993b471e1a49eb38
Normal-4787.png,Normal,Normal/images/Normal-4787.png,Normal/masks/Normal-4787.png,True,G13815,train,10fe4d0757b6a57d677d9f606ff8261b,c93bd73a18497951
Normal-4788.png,Normal,Normal/images/Normal-4788.png,Normal/masks/Normal-4788.png,True,G13816,train,4dd57732c4a246de75d9d2f980f1ecf8,947e47530b4b6b2c
Normal-4789.png,Normal,Normal/images/Normal-4789.png,Normal/masks/Normal-4789.png,True,G13817,train,03c37d08b05652fd1c4d706dd469e350,dd6c42831b4d4f1d
Normal-479.png,Normal,Normal/images/Normal-479.png,Normal/masks/Normal-479.png,True,G13818,train,52674daecac12e38d310636c22f39e6a,81b2c71eecc939c9
Normal-4790.png,Normal,Normal/images/Normal-4790.png,Normal/masks/Normal-4790.png,True,G13819,train,4ec1b0983115e13c9bce96cd26a87455,886d85871b075f1f
Normal-4791.png,Normal,Normal/images/Normal-4791.png,Normal/masks/Normal-4791.png,True,G13820,train,52ae90395f9cf3b76f4ecf41a7a387b0,817e869a5f8939c5
Normal-4792.png,Normal,Normal/images/Normal-4792.png,Normal/masks/Normal-4792.png,True,G13821,train,978d8dfe8fdfa7d06a9ff8d4fd2fa2a6,dc2d42470f0a7d3d
Normal-4793.png,Normal,Normal/images/Normal-4793.png,Normal/masks/Normal-4793.png,True,G13822,train,dbd565ab9153dc466b04bdcec7fe96cf,997dc71ec51a144e
Normal-4794.png,Normal,Normal/images/Normal-4794.png,Normal/masks/Normal-4794.png,True,G13823,train,7defd7f99b28d3bb15065a07300c41b7,dc2a530f4e0d0daf
Normal-4795.png,Normal,Normal/images/Normal-4795.png,Normal/masks/Normal-4795.png,True,G13824,test,9ef07663b93088473a1d707063d3b8e7,d4782b8b3db4e470
Normal-4796.png,Normal,Normal/images/Normal-4796.png,Normal/masks/Normal-4796.png,True,G13825,train,79df13f3ceed01b29a8b1697f864d653,987907161f8f5aca
Normal-4797.png,Normal,Normal/images/Normal-4797.png,Normal/masks/Normal-4797.png,True,G13826,test,62a2a193269ad3f03180fcba4f87ed04,dd2e520b1d8d2e3c
Normal-4798.png,Normal,Normal/images/Normal-4798.png,Normal/masks/Normal-4798.png,True,G13827,train,14547ee80c785d2a4dfbf78c1bb666cb,d8790586161e5f5e
Normal-4799.png,Normal,Normal/images/Normal-4799.png,Normal/masks/Normal-4799.png,True,G13828,train,1cdd8a0fc4701896ad4edcef1a98073f,997b07021b4dfa3a
Normal-48.png,Normal,Normal/images/Normal-48.png,Normal/masks/Normal-48.png,True,G13829,val,4d86586078e8ccb7ff9fefb6b8d3aec9,d03d42471d0b6f3d
Normal-480.png,Normal,Normal/images/Normal-480.png,Normal/masks/Normal-480.png,True,G13830,train,b98beb1728b61f609cd2ce55ea7cbc15,9168c61aba6f383d
Normal-4800.png,Normal,Normal/images/Normal-4800.png,Normal/masks/Normal-4800.png,True,G13831,val,11988106b3d1695a53fb1297e04ba4a3,996e075a1b455f68
Normal-4801.png,Normal,Normal/images/Normal-4801.png,Normal/masks/Normal-4801.png,True,G13832,train,302084e8dda6e5cdf780f1ba78333a31,817986861a4f5f5b
Normal-4802.png,Normal,Normal/images/Normal-4802.png,Normal/masks/Normal-4802.png,True,G13833,test,dc0c19d38d26c83ca5a2cc56ba0ec29e,917a1e161a996df1
Normal-4803.png,Normal,Normal/images/Normal-4803.png,Normal/masks/Normal-4803.png,True,G13834,val,1a35f390029fdc6d8951620a88b051b1,c02c3dc93d8c3e9e
Normal-4804.png,Normal,Normal/images/Normal-4804.png,Normal/masks/Normal-4804.png,True,G13835,train,9aed10371c0eb77add6f84df411abacd,9979871e184f5ab8
Normal-4805.png,Normal,Normal/images/Normal-4805.png,Normal/masks/Normal-4805.png,True,G13836,train,2e3e137f4bc010562784697f3042265f,cd2ed00b174a4f5d
Normal-4806.png,Normal,Normal/images/Normal-4806.png,Normal/masks/Normal-4806.png,True,G13837,train,d384f545cd0ce863a275b446b0407e99,dc3cc3494c0e2de7
Normal-4807.png,Normal,Normal/images/Normal-4807.png,Normal/masks/Normal-4807.png,True,G13838,train,42737e4eac93da481197ba83752995f3,d93bd61a19591b54
Normal-4808.png,Normal,Normal/images/Normal-4808.png,Normal/masks/Normal-4808.png,True,G13839,train,0abb92856323a0d8125fc911a0a28d4b,992e06199f2cb8ed
Normal-4809.png,Normal,Normal/images/Normal-4809.png,Normal/masks/Normal-4809.png,True,G13840,train,33bf2d18e484b909a83b0c669d810de9,98e753595f58451a
Normal-481.png,Normal,Normal/images/Normal-481.png,Normal/masks/Normal-481.png,True,G13841,train,35cb946f5d942caf5d1c6d9de7a9f7dd,d43b831f1d5c3c94
Normal-4810.png,Normal,Normal/images/Normal-4810.png,Normal/masks/Normal-4810.png,True,G13842,train,8a1c73f8515680329ad6ccf91567980d,9968c61a1b4e79b5
Normal-4811.png,Normal,Normal/images/Normal-4811.png,Normal/masks/Normal-4811.png,True,G13843,train,a1babf6d79520c92fad8bad0fee23fe8,993d061b1a4e39f9
Normal-4812.png,Normal,Normal/images/Normal-4812.png,Normal/masks/Normal-4812.png,True,G13844,train,af13539bea058ddcb551421573a96876,dd7d061b19443b59
Normal-4813.png,Normal,Normal/images/Normal-4813.png,Normal/masks/Normal-4813.png,True,G13845,train,ca5cc3ee8cb95606df9e1aedb7a0a1b4,d52ec7180c6e2a7d
Normal-4814.png,Normal,Normal/images/Normal-4814.png,Normal/masks/Normal-4814.png,True,G13846,val,200faadbc71c9205c06c0a37c17577bd,912e461e195f5c5d
Normal-4815.png,Normal,Normal/images/Normal-4815.png,Normal/masks/Normal-4815.png,True,G13847,test,5cd6d458d3092a279e7aceee0782926a,91794e171f461d39
Normal-4816.png,Normal,Normal/images/Normal-4816.png,Normal/masks/Normal-4816.png,True,G13848,train,e58b296851baad9cea5739011c456556,c1395f1e181b4d6d
Normal-4817.png,Normal,Normal/images/Normal-4817.png,Normal/masks/Normal-4817.png,True,G13849,train,2da3d6fe1bea8e873f4093e876562c09,d52fc25f1c486f18
Normal-4818.png,Normal,Normal/images/Normal-4818.png,Normal/masks/Normal-4818.png,True,G13850,train,99f496eb7ecca8dd80d9b354a2454d1d,9c2c821b871f595f
Normal-4819.png,Normal,Normal/images/Normal-4819.png,Normal/masks/Normal-4819.png,True,G13851,val,f168a4235a7b661334ae45d57ff7a45e,b93d87961e8d0d1c
Normal-482.png,Normal,Normal/images/Normal-482.png,Normal/masks/Normal-482.png,True,G13852,val,6f42df080baa145fddc2b11bab0caee7,957f8f1a8d0cc8c3
Normal-4820.png,Normal,Normal/images/Normal-4820.png,Normal/masks/Normal-4820.png,True,G13853,test,a5c88f21dfc55e5883d3c5c03fa021e9,995d461e79e9b0c4
Normal-4821.png,Normal,Normal/images/Normal-4821.png,Normal/masks/Normal-4821.png,True,G13854,test,6b134f1c1907c50917c70935e2862a96,dd2ec21f861aac4e
Normal-4822.png,Normal,Normal/images/Normal-4822.png,Normal/masks/Normal-4822.png,True,G13855,train,36a1227052a67f4ae014a3f1630100fc,d47e41534b5d4c55
Normal-4823.png,Normal,Normal/images/Normal-4823.png,Normal/masks/Normal-4823.png,True,G13856,train,76daf762569e123b7fe360a43d1e726e,dd2cc31b1d4d196a
Normal-4824.png,Normal,Normal/images/Normal-4824.png,Normal/masks/Normal-4824.png,True,G13857,train,c42bc818930800e4801d7b585c53e425,d57dc21a180f1f1d
Normal-4825.png,Normal,Normal/images/Normal-4825.png,Normal/masks/Normal-4825.png,True,G13858,train,0304d557b06c98d7847286fdec4f111f,9178461a194f7f39
Normal-4826.png,Normal,Normal/images/Normal-4826.png,Normal/masks/Normal-4826.png,True,G13859,train,be9f1714148ae9af86b4559febab1aba,8c6ec74d4e4c6ce2
Normal-4827.png,Normal,Normal/images/Normal-4827.png,Normal/masks/Normal-4827.png,True,G13860,train,9bf31565c28a67e17e40eafb74f95375,d57955c65b514247
Normal-4828.png,Normal,Normal/images/Normal-4828.png,Normal/masks/Normal-4828.png,True,G13861,train,b5f39350e1d93f23be4371155b789249,dcbec35bd0433d40
Normal-4829.png,Normal,Normal/images/Normal-4829.png,Normal/masks/Normal-4829.png,True,G13862,val,eb766ecd76d77284f95d83e700faf2e9,cc3f420f820f3ceb
Normal-483.png,Normal,Normal/images/Normal-483.png,Normal/masks/Normal-483.png,True,G13863,val,e9a1707b1f47f9b5597fff0a8be9503c,d13dc61bbd6938c0
Normal-4830.png,Normal,Normal/images/Normal-4830.png,Normal/masks/Normal-4830.png,True,G13864,test,0369dfd69eabee1c604b1ebe68390ebb,d42ec31949ee3ce4
Normal-4831.png,Normal,Normal/images/Normal-4831.png,Normal/masks/Normal-4831.png,True,G13865,train,c2c18935de1242452335eb544719e6e3,91694e16195e7c79
Normal-4832.png,Normal,Normal/images/Normal-4832.png,Normal/masks/Normal-4832.png,True,G13866,train,631b6d3c59d5d3fb6c2077eb143969ea,897c141a793a39af
Normal-4833.png,Normal,Normal/images/Normal-4833.png,Normal/masks/Normal-4833.png,True,G13867,train,c17eacff384af9643541fccda4564012,846c2f471a5d6e4e
Normal-4834.png,Normal,Normal/images/Normal-4834.png,Normal/masks/Normal-4834.png,True,G13868,train,3bd6e155eeab646f29e2e5332979c0a1,c16e060c5dedfc85
Normal-4835.png,Normal,Normal/images/Normal-4835.png,Normal/masks/Normal-4835.png,True,G13869,train,3ac0d04b01266c6df3d28a1d22c2333c,897d87121a4f1d5d
Normal-4836.png,Normal,Normal/images/Normal-4836.png,Normal/masks/Normal-4836.png,True,G13870,train,43441fec418bb1451039d76350920f62,d42d461f0a4e7f58
Normal-4837.png,Normal,Normal/images/Normal-4837.png,Normal/masks/Normal-4837.png,True,G13871,train,743544589b6bd21d5d78cdbc0f3498f0,8179571e1a556f58
Normal-4838.png,Normal,Normal/images/Normal-4838.png,Normal/masks/Normal-4838.png,True,G13872,val,6a9f3ac5b7b7895eb68e9e4dfe224b5e,dc2f431b054d3c4f
Normal-4839.png,Normal,Normal/images/Normal-4839.png,Normal/masks/Normal-4839.png,True,G13873,test,93760eee8ac77d77621e0a2af42c047b,8178865f5b1e7949
Normal-484.png,Normal,Normal/images/Normal-484.png,Normal/masks/Normal-484.png,True,G13874,train,0d7751c9d246c73019574b9b8fcc46c7,c16cc74b8d692df0
Normal-4840.png,Normal,Normal/images/Normal-4840.png,Normal/masks/Normal-4840.png,True,G13875,test,ec30ac84fcad691c2b2fc7a5bb75ad5a,89ebc712875e9447
Normal-4841.png,Normal,Normal/images/Normal-4841.png,Normal/masks/Normal-4841.png,True,G13876,train,0ac0a09099f00fd3cd2c390a80edaa68,b82d471a055e1dcf
Normal-4842.png,Normal,Normal/images/Normal-4842.png,Normal/masks/Normal-4842.png,True,G13877,train,64e1d526da4a9a2ed84845ae0c6aef60,896b161ac91bb9ec
Normal-4843.png,Normal,Normal/images/Normal-4843.png,Normal/masks/Normal-4843.png,True,G13878,train,bb017dfc13998e6fb17a08097dae0a06,d97d465a1d491959
Normal-4844.png,Normal,Normal/images/Normal-4844.png,Normal/masks/Normal-4844.png,True,G13879,train,cd2848b8d36c54e1c18b5c07536d7fcb,9929461e1b5d5e39
Normal-4845.png,Normal,Normal/images/Normal-4845.png,Normal/masks/Normal-4845.png,True,G13880,train,e0697714b4f7fffe713f2778227db3eb,81781e5e7a59cd68
Normal-4846.png,Normal,Normal/images/Normal-4846.png,Normal/masks/Normal-4846.png,True,G13881,train,272b799d701402f97f18b138cf9abd60,d42d871f4a4a3c4f
Normal-4847.png,Normal,Normal/images/Normal-4847.png,Normal/masks/Normal-4847.png,True,G13882,train,4e4d8feec066c6dd2534f1946f6b797c,cd1cd3cb1d3c602d
Normal-4848.png,Normal,Normal/images/Normal-4848.png,Normal/masks/Normal-4848.png,True,G13883,train,a7806fe7f7f3511062f4a7cd17e9b577,8778788e61879e78
Normal-4849.png,Normal,Normal/images/Normal-4849.png,Normal/masks/Normal-4849.png,True,G13884,train,da6fcc47b08d8e62f7fc3007e3aaee47,dc2a431f08ec2ff8
Normal-485.png,Normal,Normal/images/Normal-485.png,Normal/masks/Normal-485.png,True,G13885,train,62fe50730ca38b1680c3638a90feff27,d13dc239d869b8c3
Normal-4850.png,Normal,Normal/images/Normal-4850.png,Normal/masks/Normal-4850.png,True,G13886,train,2159c6da493d0af96caf1e8599e97a4d,991d961a1b587979
Normal-4851.png,Normal,Normal/images/Normal-4851.png,Normal/masks/Normal-4851.png,True,G13887,train,ec4dd358a32690588229619eded4d0b3,d92d061e154f1d4f
Normal-4852.png,Normal,Normal/images/Normal-4852.png,Normal/masks/Normal-4852.png,True,G13888,val,0904f6ef56b654c4a2e55b594beffad9,992c0e1a696c7ced
Normal-4853.png,Normal,Normal/images/Normal-4853.png,Normal/masks/Normal-4853.png,True,G13889,test,6064dc786bf884ea2a9d09a621de75f1,cd2ad2d34a4f4a1d
Normal-4854.png,Normal,Normal/images/Normal-4854.png,Normal/masks/Normal-4854.png,True,G13890,train,cd3bbb62ef3041d54449f4935e6e830e,d43d43c30f0b872f
Normal-4855.png,Normal,Normal/images/Normal-4855.png,Normal/masks/Normal-4855.png,True,G13891,val,0340e5e4997c47731463000266d76eee,dc2d420f0cecbdf0
Normal-4856.png,Normal,Normal/images/Normal-4856.png,Normal/masks/Normal-4856.png,True,G13892,val,55fb6712def8cbae996698d5a4e6e7cb,d528c2df1f486d1c
Normal-4857.png,Normal,Normal/images/Normal-4857.png,Normal/masks/Normal-4857.png,True,G13893,test,4513c90a762b9f4c0dac38b205c7444e,9d2d461a186d7c79
Normal-4858.png,Normal,Normal/images/Normal-4858.png,Normal/masks/Normal-4858.png,True,G13894,test,4676a77752ac28c089b38d73cfe57c6b,997a471a2e69b0e5
Normal-4859.png,Normal,Normal/images/Normal-4859.png,Normal/masks/Normal-4859.png,True,G13895,train,aaf7a7505724f9c608b385ca97251a88,c03dc35f5d4a2d98
Normal-486.png,Normal,Normal/images/Normal-486.png,Normal/masks/Normal-486.png,True,G13896,train,44a9ff06569e56016a8df456fe5afd91,d17c873bbd2c2984
Normal-4860.png,Normal,Normal/images/Normal-4860.png,Normal/masks/Normal-4860.png,True,G13897,train,3468c118fd882a7a93ded420c9e8515e,997b061a1f4e18f9
Normal-4861.png,Normal,Normal/images/Normal-4861.png,Normal/masks/Normal-4861.png,True,G13898,train,4f371d65b77c3ef17abcbd124e5ff39f,9138065f197b1b7a
Normal-4862.png,Normal,Normal/images/Normal-4862.png,Normal/masks/Normal-4862.png,True,G13899,train,425af325f805eae36707c71544f4a4f3,99f9c61ed4471651
Normal-4863.png,Normal,Normal/images/Normal-4863.png,Normal/masks/Normal-4863.png,True,G13900,train,b6b478021aff4cf1c20c0cfc384d6ab1,8968171a5e1d7a3b
Normal-4864.png,Normal,Normal/images/Normal-4864.png,Normal/masks/Normal-4864.png,True,G13901,train,7b1716c22963c7b8892bebc83b2540f5,913c06471b0fee6b
Normal-4865.png,Normal,Normal/images/Normal-4865.png,Normal/masks/Normal-4865.png,True,G13902,train,a9776f79785f728b8ec2942e4cac3d36,9d3cc24b195d5a3c
Normal-4866.png,Normal,Normal/images/Normal-4866.png,Normal/masks/Normal-4866.png,True,G13903,train,c6c587420185a6a355dd512af918b294,817907161a5f5f5a
Normal-4867.png,Normal,Normal/images/Normal-4867.png,Normal/masks/Normal-4867.png,True,G13904,train,d505bb70f349797d25da3f8c8d1b508d,997f060a196d69ba
Normal-4868.png,Normal,Normal/images/Normal-4868.png,Normal/masks/Normal-4868.png,True,G13905,train,d1d335fdfb23812adb2b5e13cec9f7dd,d03d471b495f1c4d
Normal-4869.png,Normal,Normal/images/Normal-4869.png,Normal/masks/Normal-4869.png,True,G13906,val,a9d2453e598be99ed77e16af9ca557d7,dd4d431b0b4d2c5d
Normal-487.png,Normal,Normal/images/Normal-487.png,Normal/masks/Normal-487.png,True,G13907,train,30ec789661886a6fd3a4d25f29903eb9,81a9c61a9af939b5
Normal-4870.png,Normal,Normal/images/Normal-4870.png,Normal/masks/Normal-4870.png,True,G13908,train,ce676b771dfd696a779641ab14636cde,d93dc6c31ace0e19
Normal-4871.png,Normal,Normal/images/Normal-4871.png,Normal/masks/Normal-4871.png,True,G13909,val,6d51bb9eee98dd549cf0460e4f97ae21,917e46585d5b194e
Normal-4872.png,Normal,Normal/images/Normal-4872.png,Normal/masks/Normal-4872.png,True,G13910,train,25d73485b7e3d2f5a567bc27055a7021,897dc61f1c445b39
Normal-4873.png,Normal,Normal/images/Normal-4873.png,Normal/masks/Normal-4873.png,True,G13911,val,799586bd1dc5f609db85d6a66402ab46,d83987561a1d6d78
Normal-4874.png,Normal,Normal/images/Normal-4874.png,Normal/masks/Normal-4874.png,True,G13912,test,562796d149727b1dbf25c34b4a208083,b979071a455f194d
Normal-4875.png,Normal,Normal/images/Normal-4875.png,Normal/masks/Normal-4875.png,True,G13913,test,608cca01d16cd1d09554c41eea61415b,dd2a4256525b4b1f
Normal-4876.png,Normal,Normal/images/Normal-4876.png,Normal/masks/Normal-4876.png,True,G13914,test,0455b563a5629fee3571a1b5b882d5ef,dd3c424b1b066f3c
Normal-4877.png,Normal,Normal/images/Normal-4877.png,Normal/masks/Normal-4877.png,True,G13915,test,be14a92a2371054ab10c01d2da3df4d2,d46f535b4a4f090b
Normal-4878.png,Normal,Normal/images/Normal-4878.png,Normal/masks/Normal-4878.png,True,G13916,train,c38e23693947ac6f36b91d5920adee7e,d93d42131f445a3f
Normal-4879.png,Normal,Normal/images/Normal-4879.png,Normal/masks/Normal-4879.png,True,G13917,train,24a1780f9d4732160a7bc5c369cd62d7,99ad061b0f194f1f
Normal-488.png,Normal,Normal/images/Normal-488.png,Normal/masks/Normal-488.png,True,G13918,train,6cd74f3fe26aaaaa46419d5e129a88a9,816ecb4b5c4d59cc
Normal-4880.png,Normal,Normal/images/Normal-4880.png,Normal/masks/Normal-4880.png,True,G13919,train,95a923fccf1e465341d97966bbd4c4e3,8c4cc25b0d6d3ebc
Normal-4881.png,Normal,Normal/images/Normal-4881.png,Normal/masks/Normal-4881.png,True,G13920,train,bc5d9172de785493712787a3fe18e840,cd2cd44b1d4c1b9d
Normal-4882.png,Normal,Normal/images/Normal-4882.png,Normal/masks/Normal-4882.png,True,G13921,train,9fbf85487ec03d6faf5c855ba1f67a41,d965525b471b451e
Normal-4883.png,Normal,Normal/images/Normal-4883.png,Normal/masks/Normal-4883.png,True,G13922,train,416cfaaa8dd6e9bcb2cc3a67e1da9ad8,81784b3f34e66669
Normal-4884.png,Normal,Normal/images/Normal-4884.png,Normal/masks/Normal-4884.png,True,G13923,train,e996a8e2b50986c847ed52ba4e7d45b3,dd2ec21f5c68b8e0
Normal-4885.png,Normal,Normal/images/Normal-4885.png,Normal/masks/Normal-4885.png,True,G13924,train,478cf9030acba5d811ea615288f3ba89,dd7a530a1d0a1f2b
Normal-4886.png,Normal,Normal/images/Normal-4886.png,Normal/masks/Normal-4886.png,True,G13925,train,acc350aca6b290c0726ee736fa918a24,d93a86161f2d6a39
Normal-4887.png,Normal,Normal/images/Normal-4887.png,Normal/masks/Normal-4887.png,True,G13926,train,c9f7912014cc441643175eb4a33996e7,c87dc34b0f4a4b3c
Normal-4888.png,Normal,Normal/images/Normal-4888.png,Normal/masks/Normal-4888.png,True,G13927,train,6a1c2b89c1481dfee9950e2f9511d17a,dd2a431f4c4a1f4b
Normal-4889.png,Normal,Normal/images/Normal-4889.png,Normal/masks/Normal-4889.png,True,G13928,train,4321c0cc8d9d78346c7eb6acf4456a9f,f969c61a5d421b4d
Normal-489.png,Normal,Normal/images/Normal-489.png,Normal/masks/Normal-489.png,True,G13929,train,872076bfd562e9a07cc6bec1a0a34f14,986f871a4b5f5845
Normal-4890.png,Normal,Normal/images/Normal-4890.png,Normal/masks/Normal-4890.png,True,G13930,val,38e0c1569be4546380f52dbba571b583,bc7cd31b4d4a1a49
Normal-4891.png,Normal,Normal/images/Normal-4891.png,Normal/masks/Normal-4891.png,True,G13931,val,6d13728d21945a59712db882484d0d0a,89215e5e597c6939
Normal-4892.png,Normal,Normal/images/Normal-4892.png,Normal/masks/Normal-4892.png,True,G13932,test,8cf89922e8b49eef97b501707518cbaa,d62fd1570f46260e
Normal-4893.png,Normal,Normal/images/Normal-4893.png,Normal/masks/Normal-4893.png,True,G13933,val,e4805578227513602cd95dc0e1334da4,807f041e1a7b5f3a
Normal-4894.png,Normal,Normal/images/Normal-4894.png,Normal/masks/Normal-4894.png,True,G13934,train,2231c9f5a83bcea50fbe1430f880f307,8879e7d65ec04b98
Normal-4895.png,Normal,Normal/images/Normal-4895.png,Normal/masks/Normal-4895.png,True,G13935,test,208447d39aea56b40e9542c46b9e8468,91b9861312c75f1f
Normal-4896.png,Normal,Normal/images/Normal-4896.png,Normal/masks/Normal-4896.png,True,G13936,train,2e6897b641da414872fa4d84481d1824,8168861a1f5e7b5d
Normal-4897.png,Normal,Normal/images/Normal-4897.png,Normal/masks/Normal-4897.png,True,G13937,train,de60892d8261954e3e6a795f64b44413,bc2ec30f4c5c4e4e
Normal-4898.png,Normal,Normal/images/Normal-4898.png,Normal/masks/Normal-4898.png,True,G13938,train,9b74bb6e3b251e08f8c29edefe66c855,8979d6069a1aff18
Normal-4899.png,Normal,Normal/images/Normal-4899.png,Normal/masks/Normal-4899.png,True,G13939,train,5f1524a0d6e3cb6fce1dd05b489194f2,9979871e1a055779
Normal-49.png,Normal,Normal/images/Normal-49.png,Normal/masks/Normal-49.png,True,G13940,train,1333c1305303988bea59aaa4a3910173,8179c21e5b5e1b5a
Normal-490.png,Normal,Normal/images/Normal-490.png,Normal/masks/Normal-490.png,True,G13941,train,ea30b3efd37787ed9ec33361110eacb7,91be871a0d1f5996
Normal-4900.png,Normal,Normal/images/Normal-4900.png,Normal/masks/Normal-4900.png,True,G13942,train,e61f20059dc4ea848ff706b12845559e,d92e861a455e3ae5
Normal-4901.png,Normal,Normal/images/Normal-4901.png,Normal/masks/Normal-4901.png,True,G13943,train,5ea73a14fc71c9b39e087a5863e14213,d42f430d0cacff98
Normal-4902.png,Normal,Normal/images/Normal-4902.png,Normal/masks/Normal-4902.png,True,G13944,train,9578e550420cdde622faa4ce57b01de0,c87944571a5d0f5d
Normal-4903.png,Normal,Normal/images/Normal-4903.png,Normal/masks/Normal-4903.png,True,G13945,val,315ef1fccd07cac389cf28f24ed56269,f17956c25bd14ac1
Normal-4904.png,Normal,Normal/images/Normal-4904.png,Normal/masks/Normal-4904.png,True,G13946,val,9725a4f2fc72269e5bb052a23625adf2,883f87183d59195f
Normal-4905.png,Normal,Normal/images/Normal-4905.png,Normal/masks/Normal-4905.png,True,G13947,test,41777d5f7e6a60c1e21fff53eb2a9a7e,d928875b0d6979e4
Normal-4906.png,Normal,Normal/images/Normal-4906.png,Normal/masks/Normal-4906.png,True,G13948,train,73cda54ea737d55e4ff20375eabd1a6b,98fd861a955e3147
Normal-4907.png,Normal,Normal/images/Normal-4907.png,Normal/masks/Normal-4907.png,True,G13949,train,e0557a54ea7823631d914f58e737f38f,897907861e1adf4d
Normal-4908.png,Normal,Normal/images/Normal-4908.png,Normal/masks/Normal-4908.png,True,G13950,train,033e456fb4f8231d0774839b13f6c792,897d461b4c3d78e8
Normal-4909.png,Normal,Normal/images/Normal-4909.png,Normal/masks/Normal-4909.png,True,G13951,train,df9efcd644c86675e465a9be6292c7be,d56a421a4e491f6f
Normal-491.png,Normal,Normal/images/Normal-491.png,Normal/masks/Normal-491.png,True,G13952,val,aba4c148bbedcdc5126b176e1c527553,9169c61ac86979ed
Normal-4910.png,Normal,Normal/images/Normal-4910.png,Normal/masks/Normal-4910.png,True,G13953,test,5c7bbe1920da46407c9ec88bb6cea44a,d02f121c4f5b596b
Normal-4911.png,Normal,Normal/images/Normal-4911.png,Normal/masks/Normal-4911.png,True,G13954,train,f82acdd57088923e521d82c526e91922,cc2fc34f0e2d0e4e
Normal-4912.png,Normal,Normal/images/Normal-4912.png,Normal/masks/Normal-4912.png,True,G13955,test,ee52e7df29b31ca112f8c16cdbbfb847,b86f861e0e5b184f
Normal-4913.png,Normal,Normal/images/Normal-4913.png,Normal/masks/Normal-4913.png,True,G13956,train,3c3b50569ac0aa4600b3fc483c44b3bc,d93d061b1c683bf8
Normal-4914.png,Normal,Normal/images/Normal-4914.png,Normal/masks/Normal-4914.png,True,G13957,train,58e006e6bc61fd045b39b1a5f78d49e7,997987161e8f1f18
Normal-4915.png,Normal,Normal/images/Normal-4915.png,Normal/masks/Normal-4915.png,True,G13958,train,ebcdc1ad03e75e80cac0d8a54a4f7394,813806961a5ffe3b
Normal-4916.png,Normal,Normal/images/Normal-4916.png,Normal/masks/Normal-4916.png,True,G13959,test,5e63620b8d071327d0d8a44c0bfa5b6a,dd3c421b0cec2bbc
Normal-4917.png,Normal,Normal/images/Normal-4917.png,Normal/masks/Normal-4917.png,True,G13960,train,f424fc28e33615709f4b68437956fa8f,b93f861a0d1b194f
Normal-4918.png,Normal,Normal/images/Normal-4918.png,Normal/masks/Normal-4918.png,True,G13961,train,a02b40bc644ca7be157afbb3afdb41ed,993d0603194f4f7b
Normal-4919.png,Normal,Normal/images/Normal-4919.png,Normal/masks/Normal-4919.png,True,G13962,train,3c4467faa9507fdddd829b6dd4eeb304,807b17161a7b6b3a
Normal-492.png,Normal,Normal/images/Normal-492.png,Normal/masks/Normal-492.png,True,G13963,train,362ed9a48be274ab550a7bb1e6292174,d07c871add4d3896
Normal-4920.png,Normal,Normal/images/Normal-4920.png,Normal/masks/Normal-4920.png,True,G13964,train,800e4cc093a6d72e5f074e6b53eb79dd,816906125f5f7acd
Normal-4921.png,Normal,Normal/images/Normal-4921.png,Normal/masks/Normal-4921.png,True,G13965,train,1d0806e46d1f4b7470bb2bf7758c742f,c139879698471d5f
Normal-4922.png,Normal,Normal/images/Normal-4922.png,Normal/masks/Normal-4922.png,True,G13966,train,fde74d68cb3766518356257a83a08a35,a95dc61b095d3a87
Normal-4923.png,Normal,Normal/images/Normal-4923.png,Normal/masks/Normal-4923.png,True,G13967,train,4dd930a07c7396995ff5df9f30f88622,dc2843430f2d7c7d
Normal-4924.png,Normal,Normal/images/Normal-4924.png,Normal/masks/Normal-4924.png,True,G13968,train,a41ed44cf6308d8d6913f77bc783f236,8d2dc512570f174f
Normal-4925.png,Normal,Normal/images/Normal-4925.png,Normal/masks/Normal-4925.png,True,G13969,train,c1e803bf9f8639ae33ea799185ff0ac4,d46c07487c6ce8d7
Normal-4926.png,Normal,Normal/images/Normal-4926.png,Normal/masks/Normal-4926.png,True,G13970,val,1bdd3c9eb3efa2f651a59cc17bb11397,dd7e425b424b184f
Normal-4927.png,Normal,Normal/images/Normal-4927.png,Normal/masks/Normal-4927.png,True,G13971,train,2af3241eab2cd9c4aa77f8d7f42ea5d0,d429d31e1c4d1f1e
Normal-4928.png,Normal,Normal/images/Normal-4928.png,Normal/masks/Normal-4928.png,True,G13972,train,b373479848ef8f5be9047843189eca10,d13f06071f1cca79
Normal-4929.png,Normal,Normal/images/Normal-4929.png,Normal/masks/Normal-4929.png,True,G13973,train,51155ee549c37dcbf55c99d74cab5128,d17f045c7a316707
Normal-493.png,Normal,Normal/images/Normal-493.png,Normal/masks/Normal-493.png,True,G13974,val,a6a9ff4f78c8efbcd2c03e0bf51da897,d47ea739ad2460e2
Normal-4930.png,Normal,Normal/images/Normal-4930.png,Normal/masks/Normal-4930.png,True,G13975,val,acf5d775dd6b1036cd9956c0787bd42d,d42e474b5d0e4e4e
Normal-4931.png,Normal,Normal/images/Normal-4931.png,Normal/masks/Normal-4931.png,True,G13976,train,0a6ff017f658aabe4927ba92891a4ab3,897f861a1daeb886
Normal-4932.png,Normal,Normal/images/Normal-4932.png,Normal/masks/Normal-4932.png,True,G13977,train,6cc30e1bb35f7c546e81e9d33a61bd4f,c03ce7c31c0e4f3e
Normal-4933.png,Normal,Normal/images/Normal-4933.png,Normal/masks/Normal-4933.png,True,G13978,test,edf7ecdddc6344a13b837a33b017ff89,d93d860317c74f1c
Normal-4934.png,Normal,Normal/images/Normal-4934.png,Normal/masks/Normal-4934.png,True,G13979,test,a84bd30e466d4c09a9c84e754d3558d2,992c465b19ede8b8
Normal-4935.png,Normal,Normal/images/Normal-4935.png,Normal/masks/Normal-4935.png,True,G13980,train,d91c18daae73b5a7ac5a41341b20a4b4,81781e065a5fe94f
Normal-4936.png,Normal,Normal/images/Normal-4936.png,Normal/masks/Normal-4936.png,True,G13981,train,b4bc120e33ec2985da70efef6b729c26,dc2c534b57433f44
Normal-4937.png,Normal,Normal/images/Normal-4937.png,Normal/masks/Normal-4937.png,True,G13982,train,bf66fae59b10177f975a13d4ff1fe5d8,917f871a514f1a4d
Normal-4938.png,Normal,Normal/images/Normal-4938.png,Normal/masks/Normal-4938.png,True,G13983,train,cc94b6815411b01df5a30d06606bbe36,997886121bcff8b8
Normal-4939.png,Normal,Normal/images/Normal-4939.png,Normal/masks/Normal-4939.png,True,G13984,train,b8899fa5c4254c69e52f867e53fb9610,d94ec65a5d521e16
Normal-494.png,Normal,Normal/images/Normal-494.png,Normal/masks/Normal-494.png,True,G13985,train,cdb1d4e27cd733ab2ab1578510231075,d01fc61fbcc12c96
Normal-4940.png,Normal,Normal/images/Normal-4940.png,Normal/masks/Normal-4940.png,True,G13986,train,c1204f62e56379b47436bd222a5d0518,996c061a4f7e726a
Normal-4941.png,Normal,Normal/images/Normal-4941.png,Normal/masks/Normal-4941.png,True,G13987,train,79229facfeaba89bc590d8e49f5f5fe5,dd28d75f5c485858
Normal-4942.png,Normal,Normal/images/Normal-4942.png,Normal/masks/Normal-4942.png,True,G13988,val,a5f586df673d1288efcc5767889dc8b9,d87b05861e0a5f5e
Normal-4943.png,Normal,Normal/images/Normal-4943.png,Normal/masks/Normal-4943.png,True,G13989,val,1f516b02f88fd9e02b2e586f7a7bb23c,d43dc2031d4b6f3a
Normal-4944.png,Normal,Normal/images/Normal-4944.png,Normal/masks/Normal-4944.png,True,G13990,test,7f0ff42e59db4240dfd5979fc4747f6f,dd38d30b5e0b1d29
Normal-4945.png,Normal,Normal/images/Normal-4945.png,Normal/masks/Normal-4945.png,True,G13991,test,ebfe0a5737f2639376c615a408584ee1,8879d71a1a453b3d
Normal-4946.png,Normal,Normal/images/Normal-4946.png,Normal/masks/Normal-4946.png,True,G13992,train,5b26c820a3ee50bb6422d5cbd0f3a2fd,dd2cd04b1d474f1c
Normal-4947.png,Normal,Normal/images/Normal-4947.png,Normal/masks/Normal-4947.png,True,G13993,train,e64ae4069ab79df820b44eca6f84ba5d,dcacd357d00b9c45
Normal-4948.png,Normal,Normal/images/Normal-4948.png,Normal/masks/Normal-4948.png,True,G13994,train,fa3d6db3704fe9fc5250f1806a028e2a,d02b571e3b9a39b0
Normal-4949.png,Normal,Normal/images/Normal-4949.png,Normal/masks/Normal-4949.png,True,G13995,val,7be211771ce1213edcc36218e2a9b7cb,817c065e7b7866e8
Normal-495.png,Normal,Normal/images/Normal-495.png,Normal/masks/Normal-495.png,True,G13996,train,46bc4fd61f593bc1e350285c13cb632a,9129c63a99693dcd
Normal-4950.png,Normal,Normal/images/Normal-4950.png,Normal/masks/Normal-4950.png,True,G13997,test,8527846cfcf782bca6719e8118018e50,817857171a5f699a
Normal-4951.png,Normal,Normal/images/Normal-4951.png,Normal/masks/Normal-4951.png,True,G13998,train,c77e6745bd1c76c7024ce170503aae90,d429873f080f7d4e
Normal-4952.png,Normal,Normal/images/Normal-4952.png,Normal/masks/Normal-4952.png,True,G13999,train,757f215762d57fe7a4588dbbea033cf2,813d4e0efb49ed14
Normal-4953.png,Normal,Normal/images/Normal-4953.png,Normal/masks/Normal-4953.png,True,G14000,train,2da132539e5c1269ae246182db2bf705,fd48431b4d0d1c6f
Normal-4954.png,Normal,Normal/images/Normal-4954.png,Normal/masks/Normal-4954.png,True,G14001,train,5c8bc543549bd3d4ee2fa8912f136b5c,9979471e1e497b18
Normal-4955.png,Normal,Normal/images/Normal-4955.png,Normal/masks/Normal-4955.png,True,G14002,train,4403050357576ef5a77ccdc56caa2220,ca3dd15d5e5c1a0a
Normal-4956.png,Normal,Normal/images/Normal-4956.png,Normal/masks/Normal-4956.png,True,G14003,train,553d1a7387fa9bf8c912b661c9e8758e,d4adc651974f1d44
Normal-4957.png,Normal,Normal/images/Normal-4957.png,Normal/masks/Normal-4957.png,True,G14004,train,1cfbbaf622d7579b03923a4162b824cc,9959c61e584f5e58
Normal-4958.png,Normal,Normal/images/Normal-4958.png,Normal/masks/Normal-4958.png,True,G14005,train,efd8b9df1f6b63eed00d4619c01c3494,81799e06fa1bf80b
Normal-4959.png,Normal,Normal/images/Normal-4959.png,Normal/masks/Normal-4959.png,True,G14006,train,94b337b56966dccb529f4c20ff5e50b5,9d6cc61a1b4f2e38
Normal-496.png,Normal,Normal/images/Normal-496.png,Normal/masks/Normal-496.png,True,G14007,train,0ee24f0e514d6684ee136a0f9598f805,81abe61abcc83f94
Normal-4960.png,Normal,Normal/images/Normal-4960.png,Normal/masks/Normal-4960.png,True,G14008,train,8598a77d22df4073343a88f747a513e6,c93bc61e94ce1b98
Normal-4961.png,Normal,Normal/images/Normal-4961.png,Normal/masks/Normal-4961.png,True,G14009,train,596f4b4f1f9ab205a71aa1100a6600fb,d928560a1d5c6b7d
Normal-4962.png,Normal,Normal/images/Normal-4962.png,Normal/masks/Normal-4962.png,True,G14010,val,278e3c0b31038ecee26c08099b3e021d,dc2d424f1e4a5d3c
Normal-4963.png,Normal,Normal/images/Normal-4963.png,Normal/masks/Normal-4963.png,True,G14011,test,ababb5519d616e3014d7512baef3c136,816d965a5f595a49
Normal-4964.png,Normal,Normal/images/Normal-4964.png,Normal/masks/Normal-4964.png,True,G14012,val,c88fc1436aca83388496718927b3bb3e,d97b461f040e196f
Normal-4965.png,Normal,Normal/images/Normal-4965.png,Normal/masks/Normal-4965.png,True,G14013,train,9506ce651ebe2bbb5e293f085683fc22,d42dc34b4b4f6f08
Normal-4966.png,Normal,Normal/images/Normal-4966.png,Normal/masks/Normal-4966.png,True,G14014,val,f2c1a6190c175432e3862bec792153f7,917a165a5b3b3a8a
Normal-4967.png,Normal,Normal/images/Normal-4967.png,Normal/masks/Normal-4967.png,True,G14015,train,8a00a0328c825fc23d28a1f5d295bf0d,997946161f4d1f19
Normal-4968.png,Normal,Normal/images/Normal-4968.png,Normal/masks/Normal-4968.png,True,G14016,train,09b9aac0eceadeaad3899af40aa32f27,897987161f5e1e1a
Normal-4969.png,Normal,Normal/images/Normal-4969.png,Normal/masks/Normal-4969.png,True,G14017,test,36dde836bbcea1dbbd764bac1dfe8557,9969c71611855f1f
Normal-497.png,Normal,Normal/images/Normal-497.png,Normal/masks/Normal-497.png,True,G14018,test,d51de873a1c60218bccbe0fe230a8f3c,dd2dc30b0d4d3d4c
Normal-4970.png,Normal,Normal/images/Normal-4970.png,Normal/masks/Normal-4970.png,True,G14019,train,aaaed1c67743924d60161abd739bdca6,d53d52c30b0c6d6b
Normal-4971.png,Normal,Normal/images/Normal-4971.png,Normal/masks/Normal-4971.png,True,G14020,train,34463a6ed059acf4e6a6b485d5ab221a,9969061a595f58f9
Normal-4972.png,Normal,Normal/images/Normal-4972.png,Normal/masks/Normal-4972.png,True,G14021,train,12c61c73b45609ce64dca63217f48ffc,95680f1619796d6d
Normal-4973.png,Normal,Normal/images/Normal-4973.png,Normal/masks/Normal-4973.png,True,G14022,train,3db85d96f93877bb9181aaffa81b3e36,997a57161a5f5a18
Normal-4974.png,Normal,Normal/images/Normal-4974.png,Normal/masks/Normal-4974.png,True,G14023,train,0f00bca912af8e7841fcfd949cec9e35,d92b441e5e4e1a7a
Normal-4975.png,Normal,Normal/images/Normal-4975.png,Normal/masks/Normal-4975.png,True,G14024,train,d9312a3e22a519e9353f4cee7e519b97,8169c6121b5f7b5a
Normal-4976.png,Normal,Normal/images/Normal-4976.png,Normal/masks/Normal-4976.png,True,G14025,train,b812eee12c11116a61ff48d5eddca0b1,8879071e1e797969
Normal-4977.png,Normal,Normal/images/Normal-4977.png,Normal/masks/Normal-4977.png,True,G14026,test,7970fc6846fe7bd4f6b243dfe3717514,d929861f974e185b
Normal-4978.png,Normal,Normal/images/Normal-4978.png,Normal/masks/Normal-4978.png,True,G14027,train,9e5cd44ff272baed6456b30df2d5f829,cc3c43034ce4be7d
Normal-4979.png,Normal,Normal/images/Normal-4979.png,Normal/masks/Normal-4979.png,True,G14028,train,c4acf2fce37f35144e724f4af9230df8,dd6a530f283d283e
Normal-498.png,Normal,Normal/images/Normal-498.png,Normal/masks/Normal-498.png,True,G14029,train,92a1a35ae3031b902de4f7e1873f8721,905ede1e950796a6
Normal-4980.png,Normal,Normal/images/Normal-4980.png,Normal/masks/Normal-4980.png,True,G14030,val,0e0b5ee725ec3545acf522863898528f,d92987161f491f5c
Normal-4981.png,Normal,Normal/images/Normal-4981.png,Normal/masks/Normal-4981.png,True,G14031,train,3475e03225e2f7d77ce12493d673d356,996d861e0d194d5f
Normal-4982.png,Normal,Normal/images/Normal-4982.png,Normal/masks/Normal-4982.png,True,G14032,val,9ce939ac6fde3dff2f171d7ab5944adf,d93d06061b4f4e1f
Normal-4983.png,Normal,Normal/images/Normal-4983.png,Normal/masks/Normal-4983.png,True,G14033,val,a2822d3c26b34372ea218a30b6e6b899,d53942071e784f3d
Normal-4984.png,Normal,Normal/images/Normal-4984.png,Normal/masks/Normal-4984.png,True,G14034,test,ebf995f786e560d0a1ba29ce48cfbc2d,d43d43420a4bef1f
Normal-4985.png,Normal,Normal/images/Normal-4985.png,Normal/masks/Normal-4985.png,True,G14035,train,f9665bd84e710dfae14e2b414848cf40,9928161f195e7979
Normal-4986.png,Normal,Normal/images/Normal-4986.png,Normal/masks/Normal-4986.png,True,G14036,test,919ec9644ad5221525d619e1d71f7d54,887f861a1f1e1a4f
Normal-4987.png,Normal,Normal/images/Normal-4987.png,Normal/masks/Normal-4987.png,True,G14037,train,68f5bc4cf3aaf4ba07cbf4227094b8a8,dd3d06431f0c4c6f
Normal-4988.png,Normal,Normal/images/Normal-4988.png,Normal/masks/Normal-4988.png,True,G14038,train,4555e0b5441d084fe749258b222c1225,887f471a4d1ed586
Normal-4989.png,Normal,Normal/images/Normal-4989.png,Normal/masks/Normal-4989.png,True,G14039,train,3c67a4e0be15281a848ec8be9a887ea3,817f06121b5f1e5b
Normal-499.png,Normal,Normal/images/Normal-499.png,Normal/masks/Normal-499.png,True,G14040,train,329fc955b4204842ec0fa1ba7366e12e,946f870f8d4d6cc4
Normal-4990.png,Normal,Normal/images/Normal-4990.png,Normal/masks/Normal-4990.png,True,G14041,train,bf5481ce74200a85549418684a145cb7,d43dc2431f8b4a2f
Normal-4991.png,Normal,Normal/images/Normal-4991.png,Normal/masks/Normal-4991.png,True,G14042,train,4f858880749f15088614088369ff9991,997f861a093b6d64
Normal-4992.png,Normal,Normal/images/Normal-4992.png,Normal/masks/Normal-4992.png,True,G14043,train,f70f8353e9c9a9d9252e214d929854a2,d17e561b191f091d
Normal-4993.png,Normal,Normal/images/Normal-4993.png,Normal/masks/Normal-4993.png,True,G14044,train,77fe31351517d9ba9f2b0b06acafdd90,9968071a5a5f7d4c
Normal-4994.png,Normal,Normal/images/Normal-4994.png,Normal/masks/Normal-4994.png,True,G14045,train,b8d27f823bea9d2a1e42a2e060a12d92,d91a96961a2dad35
Normal-4995.png,Normal,Normal/images/Normal-4995.png,Normal/masks/Normal-4995.png,True,G14046,val,3c7375a2696614310e71040b3d4f8de7,d63e6878692b2535
Normal-4996.png,Normal,Normal/images/Normal-4996.png,Normal/masks/Normal-4996.png,True,G14047,val,69bb3bd5de04a8051971a3b5450b5bd2,c43dcb1e130a2fb9
Normal-4997.png,Normal,Normal/images/Normal-4997.png,Normal/masks/Normal-4997.png,True,G14048,test,6754f85ed05325d9b05a5871acd02086,d979041e5d6939e1
Normal-4998.png,Normal,Normal/images/Normal-4998.png,Normal/masks/Normal-4998.png,True,G14049,train,b6d1e6fecfefb52c48ccf3a4ce87598d,dd2956461e544b5d
Normal-4999.png,Normal,Normal/images/Normal-4999.png,Normal/masks/Normal-4999.png,True,G14050,train,e90c8a5d9a15a989d4c27976ea2490f7,91791a17b9fba181
Normal-5.png,Normal,Normal/images/Normal-5.png,Normal/masks/Normal-5.png,True,G14051,train,577c7ebb783331dd715bc9383ce3ab1c,f92c861b1f451b1d
Normal-50.png,Normal,Normal/images/Normal-50.png,Normal/masks/Normal-50.png,True,G14052,train,fd38d73681a57dbfa322b1e92178a868,dd2cd0021bed2c6f
Normal-500.png,Normal,Normal/images/Normal-500.png,Normal/masks/Normal-500.png,True,G14053,val,1e2bb1d37a42b0067fbf97f86b2cfb41,d06ac74ad0c579e5
Normal-5000.png,Normal,Normal/images/Normal-5000.png,Normal/masks/Normal-5000.png,True,G14054,test,127dade236cd897c41e4b45c8906c1db,a929861e1b5d5f1a
Normal-5001.png,Normal,Normal/images/Normal-5001.png,Normal/masks/Normal-5001.png,True,G14055,test,82cce4e9da32a5cccc030e6089e6a7bb,d86d45161a4f4f3c
Normal-5002.png,Normal,Normal/images/Normal-5002.png,Normal/masks/Normal-5002.png,True,G14056,train,bd260aa402c08ac190f0216a7dd002e3,882ac55e1ed55f38
Normal-5003.png,Normal,Normal/images/Normal-5003.png,Normal/masks/Normal-5003.png,True,G14057,train,0bbc08b3dac5ee738fc3d0abeae6ca0c,d03d43431d0f4f2f
Normal-5004.png,Normal,Normal/images/Normal-5004.png,Normal/masks/Normal-5004.png,True,G14058,val,8d6eab437dd6b7e9f709450a4ed18978,dd2e425b054d3f43
Normal-5005.png,Normal,Normal/images/Normal-5005.png,Normal/masks/Normal-5005.png,True,G14059,train,25d96ae9ab29ebdc021fd7723bff90ba,dd2c82971f444f2b
Normal-5006.png,Normal,Normal/images/Normal-5006.png,Normal/masks/Normal-5006.png,True,G14060,train,954e78ce6fc8c83ce59d9c61c49b6ee2,d969c69e1a457e18
Normal-5007.png,Normal,Normal/images/Normal-5007.png,Normal/masks/Normal-5007.png,True,G14061,train,a351eb0f5ac853222332c23ae06faf0c,d969841e1f055f39
Normal-5008.png,Normal,Normal/images/Normal-5008.png,Normal/masks/Normal-5008.png,True,G14062,train,e8b05e297d0e3158203bed25463fcbef,897d06861979ede1
Normal-5009.png,Normal,Normal/images/Normal-5009.png,Normal/masks/Normal-5009.png,True,G14063,train,e4b25b78b683dcefbe7e7a3d54f377cb,9c7dd0535b430b1b
Normal-501.png,Normal,Normal/images/Normal-501.png,Normal/masks/Normal-501.png,True,G14064,train,290b66ccb97ff83fd38d90994d9c434e,d46cc71f994d29c4
Normal-5010.png,Normal,Normal/images/Normal-5010.png,Normal/masks/Normal-5010.png,True,G14065,train,2700b45a7d03ae29cd8f6d6e6fef9578,dd3ad25f544b1948
Normal-5011.png,Normal,Normal/images/Normal-5011.png,Normal/masks/Normal-5011.png,True,G14066,train,f17c358fcdfd811a3a07a931a9be8cd1,8969861c1add7a7a
Normal-5012.png,Normal,Normal/images/Normal-5012.png,Normal/masks/Normal-5012.png,True,G14067,test,aa5a66bb7ceb412fd793944fcdbd5f9d,d179464619496f3e
Normal-5013.png,Normal,Normal/images/Normal-5013.png,Normal/masks/Normal-5013.png,True,G14068,train,207f088935f416077322ca279e787956,d93d461a1d4c2b3d
Normal-5014.png,Normal,Normal/images/Normal-5014.png,Normal/masks/Normal-5014.png,True,G14069,train,7d53775e1099e734621addde615ddca1,d935c61ec01f594e
Normal-5015.png,Normal,Normal/images/Normal-5015.png,Normal/masks/Normal-5015.png,True,G14070,test,a01dd4f1f8ebf72000485fabcdb65a29,dc2a434f4c0e4d6f
Normal-5016.png,Normal,Normal/images/Normal-5016.png,Normal/masks/Normal-5016.png,True,G14071,train,5505e653a4d89ef65efa5af62168003d,897f161a5d5b08ae
Normal-5017.png,Normal,Normal/images/Normal-5017.png,Normal/masks/Normal-5017.png,True,G14072,train,23a3c6c0d0912c2183700222d325cf49,d13f0618384bf979
Normal-5018.png,Normal,Normal/images/Normal-5018.png,Normal/masks/Normal-5018.png,True,G14073,test,2aef2f8f857d6cac549512d41c11badd,d13b52825f192d2f
Normal-5019.png,Normal,Normal/images/Normal-5019.png,Normal/masks/Normal-5019.png,True,G14074,val,ed4d95b1fb6ee7feccafa842c9f2966b,d52ac2d74f1c4c4d
Normal-502.png,Normal,Normal/images/Normal-502.png,Normal/masks/Normal-502.png,True,G14075,val,0548148a3ae6daa1bffab2a2f411e69b,916cc75a896d38b5
Normal-5020.png,Normal,Normal/images/Normal-5020.png,Normal/masks/Normal-5020.png,True,G14076,train,cea8e3ed922f996b52e51774eb46f251,d97b469e5e405b11
Normal-5021.png,Normal,Normal/images/Normal-5021.png,Normal/masks/Normal-5021.png,True,G14077,train,0f1d85942ae1894ccccd590b81a5cd34,dc2dd14b0e2e2e2e
Normal-5022.png,Normal,Normal/images/Normal-5022.png,Normal/masks/Normal-5022.png,True,G14078,train,237faf511a602ce08d327b67f183c09f,8178fe861a1b155f
Normal-5023.png,Normal,Normal/images/Normal-5023.png,Normal/masks/Normal-5023.png,True,G14079,train,771427e8b8effa18c71f8a788639f3d5,cc2e070b1f4d6bd4
Normal-5024.png,Normal,Normal/images/Normal-5024.png,Normal/masks/Normal-5024.png,True,G14080,train,d57302ec8e32612f985dca3ab9526dec,987dc5961b055e1e
Normal-5025.png,Normal,Normal/images/Normal-5025.png,Normal/masks/Normal-5025.png,True,G14081,train,e2dd63e405fede87727948fdc2aaeb2e,d52e070e1c7e54ea
Normal-5026.png,Normal,Normal/images/Normal-5026.png,Normal/masks/Normal-5026.png,True,G14082,train,d75a1eeabf3b61f98c2aee1fee5b8932,897d861e4a1f7869
Normal-5027.png,Normal,Normal/images/Normal-5027.png,Normal/masks/Normal-5027.png,True,G14083,val,b2efcd05510262c29e69d182cb140927,d979461e190d2f39
Normal-5028.png,Normal,Normal/images/Normal-5028.png,Normal/masks/Normal-5028.png,True,G14084,train,20215d32aeebcbbd90464f103d85c78c,ec2cd293c3c10b9f
Normal-5029.png,Normal,Normal/images/Normal-5029.png,Normal/masks/Normal-5029.png,True,G14085,test,6ff9f2104bfccf1e11493d12955014f7,dd3cc24b194e1f98
Normal-503.png,Normal,Normal/images/Normal-503.png,Normal/masks/Normal-503.png,True,G14086,train,10d0f221528fe210b7d9ceb8ebfce9f7,912cc31bcd6d3cc9
Normal-5030.png,Normal,Normal/images/Normal-5030.png,Normal/masks/Normal-5030.png,True,G14087,train,e155520ef72a529838800439a8860a93,d92dc51616474f1e
Normal-5031.png,Normal,Normal/images/Normal-5031.png,Normal/masks/Normal-5031.png,True,G14088,train,0dee4de682b9a89b285a8571ba6134da,d92c061f5e38f8e1
Normal-5032.png,Normal,Normal/images/Normal-5032.png,Normal/masks/Normal-5032.png,True,G14089,train,88a74eb5a4c98919c15283500ae8a408,dd3d8608d45f984e
Normal-5033.png,Normal,Normal/images/Normal-5033.png,Normal/masks/Normal-5033.png,True,G14090,test,356c378849a2f2b5a9c05b83ee178443,d42ec35fd44d5c44
Normal-5034.png,Normal,Normal/images/Normal-5034.png,Normal/masks/Normal-5034.png,True,G14091,train,f3b9c19c7c0c502169c1483558c35f71,cc2d42591f4a3dbc
Normal-5035.png,Normal,Normal/images/Normal-5035.png,Normal/masks/Normal-5035.png,True,G14092,train,65ef827c1ed7a1d4b0141754ba22f5c7,dd2c421f4c4b3de2
Normal-5036.png,Normal,Normal/images/Normal-5036.png,Normal/masks/Normal-5036.png,True,G14093,train,13908988033a2e6a73c171c1e6ab1cb3,d43c47d10b4e6c3b
Normal-5037.png,Normal,Normal/images/Normal-5037.png,Normal/masks/Normal-5037.png,True,G14094,val,36f9a5e5704cb825156f1128694ff2ca,816a865b5c5b396d
Normal-5038.png,Normal,Normal/images/Normal-5038.png,Normal/masks/Normal-5038.png,True,G14095,val,295a6606360ae310c893142f92407083,ed3d441317451b4f
Normal-5039.png,Normal,Normal/images/Normal-5039.png,Normal/masks/Normal-5039.png,True,G14096,val,b231943d1e8901df3a620785696ea69c,cc3fc30f0c0b0cef
Normal-504.png,Normal,Normal/images/Normal-504.png,Normal/masks/Normal-504.png,True,G14097,test,a9713ef6cd764c5ae5a5336995c31ab9,94788739ad6c3cd4
Normal-5040.png,Normal,Normal/images/Normal-5040.png,Normal/masks/Normal-5040.png,True,G14098,train,2baf18138fe642493c5b12f12e97ece1,9979071ed51e5817
Normal-5041.png,Normal,Normal/images/Normal-5041.png,Normal/masks/Normal-5041.png,True,G14099,test,1d4a3044f3b5151a811f37af45395848,81795786185fa97a
Normal-5042.png,Normal,Normal/images/Normal-5042.png,Normal/masks/Normal-5042.png,True,G14100,train,b40d5f9776c402ec54318d4f04f84a88,91e844df798f2991
Normal-5043.png,Normal,Normal/images/Normal-5043.png,Normal/masks/Normal-5043.png,True,G14101,train,42855f0fb771ed8ba5393e043b842f65,997c0686125bcfba
Normal-5044.png,Normal,Normal/images/Normal-5044.png,Normal/masks/Normal-5044.png,True,G14102,val,0a7b6ed021e8cb6d3db11c4b86dedccf,813a16185f7d7b61
Normal-5045.png,Normal,Normal/images/Normal-5045.png,Normal/masks/Normal-5045.png,True,G14103,train,20d309b8587eb525eff59d946d6478f6,d52852d35b0e4c5f
Normal-5046.png,Normal,Normal/images/Normal-5046.png,Normal/masks/Normal-5046.png,True,G14104,train,c880d15826b362648d44de52fd73930a,81697e9e75c66660
Normal-5047.png,Normal,Normal/images/Normal-5047.png,Normal/masks/Normal-5047.png,True,G14105,train,9a4e9153c7a962e4662b349054f63fd9,d53c025b190fec79
Normal-5048.png,Normal,Normal/images/Normal-5048.png,Normal/masks/Normal-5048.png,True,G14106,train,af8971e0b61f62dfa60a44f3072f7621,9178041e5b5b5b5e
Normal-5049.png,Normal,Normal/images/Normal-5049.png,Normal/masks/Normal-5049.png,True,G14107,train,b1f4c4fafbf46802cd469ee5d8cac0c5,9938171a1b6f79a8
Normal-505.png,Normal,Normal/images/Normal-505.png,Normal/masks/Normal-505.png,True,G14108,train,d9c1ca704b4455b3f9c4614d4cecd84e,917caf3a0f49384d
Normal-5050.png,Normal,Normal/images/Normal-5050.png,Normal/masks/Normal-5050.png,True,G14109,train,690e4bd6343dce858d7f453807cf66c8,911e96185b7b196b
Normal-5051.png,Normal,Normal/images/Normal-5051.png,Normal/masks/Normal-5051.png,True,G14110,train,d25917a89b11bfde27dc5cb082cb3512,916946065f19fc4f
Normal-5052.png,Normal,Normal/images/Normal-5052.png,Normal/masks/Normal-5052.png,True,G14111,val,b47f62d40f8e2e944e364fe1b548c61e,c97a87461a174b5b
Normal-5053.png,Normal,Normal/images/Normal-5053.png,Normal/masks/Normal-5053.png,True,G14112,train,be0b851836dbfa770d394900103a40ef,9879e59684c557a6
Normal-5054.png,Normal,Normal/images/Normal-5054.png,Normal/masks/Normal-5054.png,True,G14113,val,3d08c75b6c9a860a0fc1018f4fdd96fc,d939565e5c6a1d28
Normal-5055.png,Normal,Normal/images/Normal-5055.png,Normal/masks/Normal-5055.png,True,G14114,test,a76744887653f459aa30a93c973c9c04,d92f861a584f5e49
Normal-5056.png,Normal,Normal/images/Normal-5056.png,Normal/masks/Normal-5056.png,True,G14115,train,dc5d22c1506b714004c7e97d3f167a4f,9968161a195d7b1f
Normal-5057.png,Normal,Normal/images/Normal-5057.png,Normal/masks/Normal-5057.png,True,G14116,test,50332d44c8ee9d79c6ed11a8f737f42c,d97b421b19493bb8
Normal-5058.png,Normal,Normal/images/Normal-5058.png,Normal/masks/Normal-5058.png,True,G14117,train,df9f80d097e25e66a8c12c9d3bb0ec6c,996dc61e4a49394f
Normal-5059.png,Normal,Normal/images/Normal-5059.png,Normal/masks/Normal-5059.png,True,G14118,train,4496e55803b1490fb652e498cbc954a8,d42f435d0d2a38be
Normal-506.png,Normal,Normal/images/Normal-506.png,Normal/masks/Normal-506.png,True,G14119,train,49a7f878f76c44efbac656e85a3315d2,d474c31baded3824
Normal-5060.png,Normal,Normal/images/Normal-5060.png,Normal/masks/Normal-5060.png,True,G14120,train,74288fb9d749e2c525981df72b31304a,9179c6125e5f185d
Normal-5061.png,Normal,Normal/images/Normal-5061.png,Normal/masks/Normal-5061.png,True,G14121,train,58660a61dafafb5d68b754641aa90ceb,9d6e831bc7471945
Normal-5062.png,Normal,Normal/images/Normal-5062.png,Normal/masks/Normal-5062.png,True,G14122,train,90056e7c31c85560280ddfc5f7582271,886d8616594b5e7e
Normal-5063.png,Normal,Normal/images/Normal-5063.png,Normal/masks/Normal-5063.png,True,G14123,train,d3d93a0460491b52c3213c7eec8b4845,8169d6d45b515b59
Normal-5064.png,Normal,Normal/images/Normal-5064.png,Normal/masks/Normal-5064.png,True,G14124,train,5edaeaf2f33dc0b800025e2c7f61d84a,9d2f060f054f5f45
Normal-5065.png,Normal,Normal/images/Normal-5065.png,Normal/masks/Normal-5065.png,True,G14125,test,7841fe733cbb711ebf7cbbbda6b62264,807d87161e8f1e1f
Normal-5066.png,Normal,Normal/images/Normal-5066.png,Normal/masks/Normal-5066.png,True,G14126,val,3ea15a6db2f4f383cb6a2342622b3c38,996e861bc44e79e4
Normal-5067.png,Normal,Normal/images/Normal-5067.png,Normal/masks/Normal-5067.png,True,G14127,train,7111a8305170854d9c3ed818b5e82873,b979861ed51a1c4e
Normal-5068.png,Normal,Normal/images/Normal-5068.png,Normal/masks/Normal-5068.png,True,G14128,train,07ecb619c2651367ce3e07447516131d,d5708e1a8a4f1b4f
Normal-5069.png,Normal,Normal/images/Normal-5069.png,Normal/masks/Normal-5069.png,True,G14129,val,1b1e834f7c8821141a229f3d2cc86d08,d2d634c20a2dc73f
Normal-507.png,Normal,Normal/images/Normal-507.png,Normal/masks/Normal-507.png,True,G14130,train,4e0bed029bd4ee0ec324742e198cc9f8,89a9c65ede4a3968
Normal-5070.png,Normal,Normal/images/Normal-5070.png,Normal/masks/Normal-5070.png,True,G14131,train,f05b913caf5ef837b9c1488a716b8545,996cc75b414e594b
Normal-5071.png,Normal,Normal/images/Normal-5071.png,Normal/masks/Normal-5071.png,True,G14132,val,c7b08299ea1d362130832274c0e9ad39,d42ad3437d442a7d
Normal-5072.png,Normal,Normal/images/Normal-5072.png,Normal/masks/Normal-5072.png,True,G14133,test,71a5c497306f54d08fa0067911ab28e2,9178861819dfbb9a
Normal-5073.png,Normal,Normal/images/Normal-5073.png,Normal/masks/Normal-5073.png,True,G14134,train,64ae156fbd002e4bfc05c7d80b7685eb,957f860a174e1b55
Normal-5074.png,Normal,Normal/images/Normal-5074.png,Normal/masks/Normal-5074.png,True,G14135,train,1f814dd5392d411edd33fb23081d681b,996dc69a194d7a58
Normal-5075.png,Normal,Normal/images/Normal-5075.png,Normal/masks/Normal-5075.png,True,G14136,test,0956c101f0571044d76135a478f39582,897d855295c61f95
Normal-5076.png,Normal,Normal/images/Normal-5076.png,Normal/masks/Normal-5076.png,True,G14137,train,3e2fe9f3b2c9f8cae16afc79841d9312,d979c61a188d1bbc
Normal-5077.png,Normal,Normal/images/Normal-5077.png,Normal/masks/Normal-5077.png,True,G14138,test,aba29b5beb5764d3bf70edd008769567,d43d4b0e4e4f0c6e
Normal-5078.png,Normal,Normal/images/Normal-5078.png,Normal/masks/Normal-5078.png,True,G14139,train,38a3a116e7257a7db2660e72cd5fe6fa,997b061a1d4d3b39
Normal-5079.png,Normal,Normal/images/Normal-5079.png,Normal/masks/Normal-5079.png,True,G14140,train,074087ba76b5163260676d8e0d7a0190,9968071a175e575e
Normal-508.png,Normal,Normal/images/Normal-508.png,Normal/masks/Normal-508.png,True,G14141,train,9141b96abdfb10bb1703b8e928212448,d16f871a0d4d1c6d
Normal-5080.png,Normal,Normal/images/Normal-5080.png,Normal/masks/Normal-5080.png,True,G14142,val,40f3776a8d79352af16845842545ad86,d835031e4e5f1a5e
Normal-5081.png,Normal,Normal/images/Normal-5081.png,Normal/masks/Normal-5081.png,True,G14143,train,4f54b3aa5197b9b32b2bc7dbd91a6112,f95dc7161a493938
Normal-5082.png,Normal,Normal/images/Normal-5082.png,Normal/masks/Normal-5082.png,True,G14144,train,8b2ef52feb4bac7a445b18eb6af87dc7,dd3d820b1d4c7939
Normal-5083.png,Normal,Normal/images/Normal-5083.png,Normal/masks/Normal-5083.png,True,G14145,train,09b1586617652cacc777c05bfea520ee,dc2dc25751430f5d
Normal-5084.png,Normal,Normal/images/Normal-5084.png,Normal/masks/Normal-5084.png,True,G14146,train,f727cbe9536db5c0a46c3e9bb769c7f7,993b05161a5a7b79
Normal-5085.png,Normal,Normal/images/Normal-5085.png,Normal/masks/Normal-5085.png,True,G14147,train,eea2becf40205951477fd420c0996876,956a861e091f5c5f
Normal-5086.png,Normal,Normal/images/Normal-5086.png,Normal/masks/Normal-5086.png,True,G14148,train,13a71ae3259d8fcf1986a90c23c6f6da,81780e861b59ef7a
Normal-5087.png,Normal,Normal/images/Normal-5087.png,Normal/masks/Normal-5087.png,True,G14149,train,4f444720c85aa13f46a70855e4bb445b,992c161a5d5c5b7c
Normal-5088.png,Normal,Normal/images/Normal-5088.png,Normal/masks/Normal-5088.png,True,G14150,train,920177e2f1f357b0900d220ccb6af89a,9968864a1a5f4f3e
Normal-5089.png,Normal,Normal/images/Normal-5089.png,Normal/masks/Normal-5089.png,True,G14151,train,48e8e594d1801631f68109ce7fff9b97,d93f861a1d491a5d
Normal-509.png,Normal,Normal/images/Normal-509.png,Normal/masks/Normal-509.png,True,G14152,train,95a94b691a57d6321abb1e592416c3e7,986fc71a875b104f
Normal-5090.png,Normal,Normal/images/Normal-5090.png,Normal/masks/Normal-5090.png,True,G14153,train,05577e37fc566320cf9263f3830fd841,9169161c1f5a795b
Normal-5091.png,Normal,Normal/images/Normal-5091.png,Normal/masks/Normal-5091.png,True,G14154,train,3c27087a4e4fdce4fa94f847db3d04ea,cc25461a447ae5ef
Normal-5092.png,Normal,Normal/images/Normal-5092.png,Normal/masks/Normal-5092.png,True,G14155,val,98b9cf4d2fe6944b68fdc35ba1b934de,c929d41b5d461f1d
Normal-5093.png,Normal,Normal/images/Normal-5093.png,Normal/masks/Normal-5093.png,True,G14156,val,a7db3c276063e7ba95a4118f4d494d03,997b85121fc61f49
Normal-5094.png,Normal,Normal/images/Normal-5094.png,Normal/masks/Normal-5094.png,True,G14157,test,c3a31b3b2ebed8d8b714829be964cf70,d97906471b1e4f1a
Normal-5095.png,Normal,Normal/images/Normal-5095.png,Normal/masks/Normal-5095.png,True,G14158,test,54ac5845a50c14857b1b187d3e6a6434,9d2cc2134f451f5d
Normal-5096.png,Normal,Normal/images/Normal-5096.png,Normal/masks/Normal-5096.png,True,G14159,train,12a1444f59dc4a813fdffabd2f91579b,dc2ac30d1c4f5e4b
Normal-5097.png,Normal,Normal/images/Normal-5097.png,Normal/masks/Normal-5097.png,True,G14160,train,e7cf0a528bfb67ad71151ac37da09968,dc2c431a1869bcfe
Normal-5098.png,Normal,Normal/images/Normal-5098.png,Normal/masks/Normal-5098.png,True,G14161,train,928920a1e77439f231dabc793dd2fc24,997dc45b194e4f18
Normal-5099.png,Normal,Normal/images/Normal-5099.png,Normal/masks/Normal-5099.png,True,G14162,train,dd1de14d3716fa1fc8b4b04b92bc4e5f,896ad6c25b5d5a52
Normal-51.png,Normal,Normal/images/Normal-51.png,Normal/masks/Normal-51.png,True,G14163,train,8774fe287a8a8bdd62b0b269611b1ae1,d50c4f0b5d0a6e7c
Normal-510.png,Normal,Normal/images/Normal-510.png,Normal/masks/Normal-510.png,True,G14164,train,57a9929566242892b731994aacf14028,d46ea70bcd4d68c4
Normal-5100.png,Normal,Normal/images/Normal-5100.png,Normal/masks/Normal-5100.png,True,G14165,train,3cf48327dc3c660812975307bbe5800a,83788c94375e1b3b
Normal-5101.png,Normal,Normal/images/Normal-5101.png,Normal/masks/Normal-5101.png,True,G14166,train,af358624db2e566ae2f9f6723a5031e7,987dc61e18457f19
Normal-5102.png,Normal,Normal/images/Normal-5102.png,Normal/masks/Normal-5102.png,True,G14167,train,fc0a7a82f68fbcbfff5fc0895e544b77,dd2c52c34b0f4d2e
Normal-5103.png,Normal,Normal/images/Normal-5103.png,Normal/masks/Normal-5103.png,True,G10395,val,992bad65f8f8801692908cbefeb99d2b,dd2cc25b154b1f4c
Normal-5104.png,Normal,Normal/images/Normal-5104.png,Normal/masks/Normal-5104.png,True,G14168,train,ce1d09fc08612d98f287f496fed65c5e,d938c41a1d4a3df9
Normal-5105.png,Normal,Normal/images/Normal-5105.png,Normal/masks/Normal-5105.png,True,G14169,train,92556f12868614cfd6bc8acef37d32b7,dc75834eac4cac4e
Normal-5106.png,Normal,Normal/images/Normal-5106.png,Normal/masks/Normal-5106.png,True,G14170,train,5e0699d393587d712de64e5e87ee333a,917f861a1a173a4f
Normal-5107.png,Normal,Normal/images/Normal-5107.png,Normal/masks/Normal-5107.png,True,G14171,train,9b95aa234340a2ef2a79e09d8a106a31,d93f465b1d4e181c
Normal-5108.png,Normal,Normal/images/Normal-5108.png,Normal/masks/Normal-5108.png,True,G14172,val,e91ef7316935b42f4384332a845d5554,d93dc21f924f1869
Normal-5109.png,Normal,Normal/images/Normal-5109.png,Normal/masks/Normal-5109.png,True,G14173,val,4caf96b3759257b1186e566a73c8fe62,987bc51697471e94
Normal-511.png,Normal,Normal/images/Normal-511.png,Normal/masks/Normal-511.png,True,G14174,test,202343589d0a1695870f47259d952544,913fa63a951d1d85
Normal-5110.png,Normal,Normal/images/Normal-5110.png,Normal/masks/Normal-5110.png,True,G14175,test,7d44a7a0312c4c26f1c5e4e55616669d,c95956191e2d3da5
Normal-5111.png,Normal,Normal/images/Normal-5111.png,Normal/masks/Normal-5111.png,True,G14176,val,aa340faddbf9c190747b8f68c8a877bc,d42fc14b0d4e2e6e
Normal-5112.png,Normal,Normal/images/Normal-5112.png,Normal/masks/Normal-5112.png,True,G14177,test,1733483eb25c4c3f5ce0cbbee80f0c22,d03d4b535f484a7c
Normal-5113.png,Normal,Normal/images/Normal-5113.png,Normal/masks/Normal-5113.png,True,G14178,train,155b5fa5e7e66e1469be1bf59d813a50,d97906145e4f185f
Normal-5114.png,Normal,Normal/images/Normal-5114.png,Normal/masks/Normal-5114.png,True,G14179,train,515ea49e82f12ec4b02c642c85e7414f,d52a025f384959bf
Normal-5115.png,Normal,Normal/images/Normal-5115.png,Normal/masks/Normal-5115.png,True,G14180,train,77ec3d1e1a27c73b4be98d6c2c924615,997a1a193b39a7b0
Normal-5116.png,Normal,Normal/images/Normal-5116.png,Normal/masks/Normal-5116.png,True,G14181,train,face98ad613fff24a80d0a68b2497f20,807be7061e959e3a
Normal-5117.png,Normal,Normal/images/Normal-5117.png,Normal/masks/Normal-5117.png,True,G14182,test,9dcf552ae447bf0413b395596b5b6f38,987f861a091d1f57
Normal-5118.png,Normal,Normal/images/Normal-5118.png,Normal/masks/Normal-5118.png,True,G14183,train,d90206538ff46360435ecea24b98f3e0,d43dc2c35b0ccf0b
Normal-5119.png,Normal,Normal/images/Normal-5119.png,Normal/masks/Normal-5119.png,True,G14184,train,c6a4d2c088b9ebd21f5f5a65bc764545,91e946d70ac47ad3
Normal-512.png,Normal,Normal/images/Normal-512.png,Normal/masks/Normal-512.png,True,G14185,train,a89736fe3b0fd73e27e8f6100c2aca15,d4efc70bed6828c0
Normal-5120.png,Normal,Normal/images/Normal-5120.png,Normal/masks/Normal-5120.png,True,G14186,train,d0031accf51207a7e57d0393e2b589ee,d82b45061e18fb7b
Normal-5121.png,Normal,Normal/images/Normal-5121.png,Normal/masks/Normal-5121.png,True,G14187,train,0e35d7ec26d922a75495adf5fc5016fd,997ad7431b19093b
Normal-5122.png,Normal,Normal/images/Normal-5122.png,Normal/masks/Normal-5122.png,True,G14188,train,b28b6392d735eefb9c97bb3c4bc1e467,c03d4ad65d4bc22f
Normal-5123.png,Normal,Normal/images/Normal-5123.png,Normal/masks/Normal-5123.png,True,G14189,train,5d4fe0551555759745e65677356cdec2,d02d4f155b086f2f
Normal-5124.png,Normal,Normal/images/Normal-5124.png,Normal/masks/Normal-5124.png,True,G14190,train,58b16d2a6fe3de4df5cd0912d6680c3b,9b481e5a5b79c6c2
Normal-5125.png,Normal,Normal/images/Normal-5125.png,Normal/masks/Normal-5125.png,True,G14191,train,c213ff9c367a595ff68e7708ae188772,996d861acc1f1b4d
Normal-5126.png,Normal,Normal/images/Normal-5126.png,Normal/masks/Normal-5126.png,True,G14192,train,53981a49a9ed7831b6c068b0a538a65a,887b179e1d587ac1
Normal-5127.png,Normal,Normal/images/Normal-5127.png,Normal/masks/Normal-5127.png,True,G14193,val,9663e43e20a275e81aa8b8fa17a3af7c,d62f014f4e6a2e2b
Normal-5128.png,Normal,Normal/images/Normal-5128.png,Normal/masks/Normal-5128.png,True,G14194,val,4ac3364af55f0ec91bf39a25e127b7ac,8878871f1c177a1f
Normal-5129.png,Normal,Normal/images/Normal-5129.png,Normal/masks/Normal-5129.png,True,G14195,val,48accbf2ffc3248106785aca1089d62d,dd2ad21d421f5a4d
Normal-513.png,Normal,Normal/images/Normal-513.png,Normal/masks/Normal-513.png,True,G14196,test,313e75545aa1570f992e816d3f200d37,913dc71a99d93998
Normal-5130.png,Normal,Normal/images/Normal-5130.png,Normal/masks/Normal-5130.png,True,G14197,test,ba71d3284ff326948a2eb262d8add638,d52e87931b4d055e
Normal-5131.png,Normal,Normal/images/Normal-5131.png,Normal/masks/Normal-5131.png,True,G14198,train,627b59872a6e28d556f882dd816f8c62,893d4e1e386b3879
Normal-5132.png,Normal,Normal/images/Normal-5132.png,Normal/masks/Normal-5132.png,True,G14199,train,1c23baed7033808172127817ac09f184,d97b560a5c0b3b19
Normal-5133.png,Normal,Normal/images/Normal-5133.png,Normal/masks/Normal-5133.png,True,G14200,train,363c30ff73b51dc06c97581948ffafce,81fa57161e154f1b
Normal-5134.png,Normal,Normal/images/Normal-5134.png,Normal/masks/Normal-5134.png,True,G14201,train,480c81d998758af0c232012deb714c9b,dc2d420f18ec6f3c
Normal-5135.png,Normal,Normal/images/Normal-5135.png,Normal/masks/Normal-5135.png,True,G14202,train,43cc5b474f25e7b3c4b8bff23772fff4,dc2d52534b0e2f0f
Normal-5136.png,Normal,Normal/images/Normal-5136.png,Normal/masks/Normal-5136.png,True,G14203,val,75c406bba4d6320415b607734c8f9eb0,9929c71e544f5e58
Normal-5137.png,Normal,Normal/images/Normal-5137.png,Normal/masks/Normal-5137.png,True,G14204,train,921c273250b03dde71cef9368b7576a8,dc3dd2434bc20f1d
Normal-5138.png,Normal,Normal/images/Normal-5138.png,Normal/masks/Normal-5138.png,True,G14205,test,f73461c6f379b8eed501adc7db53db0e,917f861a59da7868
Normal-5139.png,Normal,Normal/images/Normal-5139.png,Normal/masks/Normal-5139.png,True,G14206,train,e1a763b6bf5d614ba67dd3a8ab78ae84,9d2dd51251471a5f
Normal-514.png,Normal,Normal/images/Normal-514.png,Normal/masks/Normal-514.png,True,G14207,test,fea82931c5a83b1866acc1cfb261ad61,c0eeeb7e4d5448c2
Normal-5140.png,Normal,Normal/images/Normal-5140.png,Normal/masks/Normal-5140.png,True,G14208,train,43cba86c7f34ec6fe808bca70a4fbbc9,9d6d861a5d4d1a4d
Normal-5141.png,Normal,Normal/images/Normal-5141.png,Normal/masks/Normal-5141.png,True,G14209,train,42812414d6d70c552223b8f0cf404377,912e061b0d4f5e5f
Normal-5142.png,Normal,Normal/images/Normal-5142.png,Normal/masks/Normal-5142.png,True,G14210,train,5787d01f45dfa8f11e1dd267540018a0,896d065e59e9b8e1
Normal-5143.png,Normal,Normal/images/Normal-5143.png,Normal/masks/Normal-5143.png,True,G14211,train,5df720116cec38d03a38d9f047caad98,986ed61b425b5d0b
Normal-5144.png,Normal,Normal/images/Normal-5144.png,Normal/masks/Normal-5144.png,True,G14212,train,ba755043d2472cabe2e95e8d61de370a,813d460f39acadb9
Normal-5145.png,Normal,Normal/images/Normal-5145.png,Normal/masks/Normal-5145.png,True,G14213,train,e0d19f93550b186723f813f47348de03,817b9606155f5e1d
Normal-5146.png,Normal,Normal/images/Normal-5146.png,Normal/masks/Normal-5146.png,True,G14214,val,856c6667f46cc1d5a20dde94b0c4e056,fd45c71e5b4f1808
Normal-5147.png,Normal,Normal/images/Normal-5147.png,Normal/masks/Normal-5147.png,True,G14215,train,08df7a7039d37c45c740d01595440167,993b46c61e545e0f
Normal-5148.png,Normal,Normal/images/Normal-5148.png,Normal/masks/Normal-5148.png,True,G14216,train,82cdef71748250b2aa96ecb30089becd,9c4d435b4c69f8f0
Normal-5149.png,Normal,Normal/images/Normal-5149.png,Normal/masks/Normal-5149.png,True,G14217,val,8f4288138542e66bfc1d43cdd72801bd,c9b9c61f9646194b
Normal-515.png,Normal,Normal/images/Normal-515.png,Normal/masks/Normal-515.png,True,G14218,test,658759c5d819f4626aef4ea261e2c465,d03de32db86c3c58
Normal-5150.png,Normal,Normal/images/Normal-5150.png,Normal/masks/Normal-5150.png,True,G14219,train,62921b3c7cb6d71aae1d28907f6dce9f,dd2c430b1e6c783d
Normal-5151.png,Normal,Normal/images/Normal-5151.png,Normal/masks/Normal-5151.png,True,G14220,train,ace4ec776f3711d9ccf09343f7ae3572,cc3d500b1fd42abd
Normal-5152.png,Normal,Normal/images/Normal-5152.png,Normal/masks/Normal-5152.png,True,G14221,train,7ec176360c94e639331f7fa3d690e865,d03dd2525e4ad25b
Normal-5153.png,Normal,Normal/images/Normal-5153.png,Normal/masks/Normal-5153.png,True,G14222,train,b0b72dbedb992a2b62db411df800fd9c,dc2dc35f544e1c58
Normal-5154.png,Normal,Normal/images/Normal-5154.png,Normal/masks/Normal-5154.png,True,G14223,train,242cb783db411e26b7c49264d49dfe51,d93cc61f114f3949
Normal-5155.png,Normal,Normal/images/Normal-5155.png,Normal/masks/Normal-5155.png,True,G14224,train,90749ab1d539a058ccf794b07dbb119c,8929061e1e5f5e5b
Normal-5156.png,Normal,Normal/images/Normal-5156.png,Normal/masks/Normal-5156.png,True,G14225,val,a5559124c6e86182cce5659fbbe4c6db,986a421f4f0f2b3d
Normal-5157.png,Normal,Normal/images/Normal-5157.png,Normal/masks/Normal-5157.png,True,G14226,train,f8c7250c38ba37954da89cf822549522,dc3cc21d534f1c4b
Normal-5158.png,Normal,Normal/images/Normal-5158.png,Normal/masks/Normal-5158.png,True,G14227,train,68da04ce57316552357b0b228a7fedde,917c4743194d3d79
Normal-5159.png,Normal,Normal/images/Normal-5159.png,Normal/masks/Normal-5159.png,True,G14228,train,07459c94993aa0cf50c2aef5c6db81d1,dd28d61b550e1dd4
Normal-516.png,Normal,Normal/images/Normal-516.png,Normal/masks/Normal-516.png,True,G14229,train,2fdcd680f9c7b6b545bdbde2e2d9c2d8,917fc61ac91d1987
Normal-5160.png,Normal,Normal/images/Normal-5160.png,Normal/masks/Normal-5160.png,True,G14230,train,9b074665d76f1e89241fb5b22f40388d,a939871e1a873d1b
Normal-5161.png,Normal,Normal/images/Normal-5161.png,Normal/masks/Normal-5161.png,True,G14231,train,e29ccc78a70b72b9fa7d3fe3700d3d0d,cd2e420f0dec2ef4
Normal-5162.png,Normal,Normal/images/Normal-5162.png,Normal/masks/Normal-5162.png,True,G14232,train,65250de3bceb65dba80ce96d2cc4b7e6,d42f535f0f483d0a
Normal-5163.png,Normal,Normal/images/Normal-5163.png,Normal/masks/Normal-5163.png,True,G14233,val,176838d4abf02eeff9847d00534ef587,cd2e07697c68f1a2
Normal-5164.png,Normal,Normal/images/Normal-5164.png,Normal/masks/Normal-5164.png,True,G14234,val,918abffa382d6c409b67c20d06d88420,8c7cc25b454b1b4f
Normal-5165.png,Normal,Normal/images/Normal-5165.png,Normal/masks/Normal-5165.png,True,G14235,test,39e8e26e29a0fc789ec7234e3ead928f,9d2c465b496e78e1
Normal-5166.png,Normal,Normal/images/Normal-5166.png,Normal/masks/Normal-5166.png,True,G14236,test,807370677a7b84f663ccf2825c77b461,8879871e1e057a5f
Normal-5167.png,Normal,Normal/images/Normal-5167.png,Normal/masks/Normal-5167.png,True,G14237,train,360463318e71e4e5e7a7d92dc30a6bc5,95186b9f60e76662
Normal-5168.png,Normal,Normal/images/Normal-5168.png,Normal/masks/Normal-5168.png,True,G14238,test,10aff4cef26540f8a964e7e463d03a72,d969461a1b4f383e
Normal-5169.png,Normal,Normal/images/Normal-5169.png,Normal/masks/Normal-5169.png,True,G14239,train,d23011b269f0f5ffcf34cbe719bcea23,cd2cc21b1f4c0efc
Normal-517.png,Normal,Normal/images/Normal-517.png,Normal/masks/Normal-517.png,True,G14240,train,987671c03742262b186790384016ef1c,91bec61ac53cb4c6
Normal-5170.png,Normal,Normal/images/Normal-5170.png,Normal/masks/Normal-5170.png,True,G14241,train,358e17ec41ed25be4fb30c94c56a2987,917b461a1b6d39b1
Normal-5171.png,Normal,Normal/images/Normal-5171.png,Normal/masks/Normal-5171.png,True,G14242,val,8c735f54f57b66f6234bd097bcb23652,dc2ac39bdd454a1c
Normal-5172.png,Normal,Normal/images/Normal-5172.png,Normal/masks/Normal-5172.png,True,G14243,train,8eae5b5e76b5ac1fe5aa4908820ee200,fd38d60b1d0819bd
Normal-5173.png,Normal,Normal/images/Normal-5173.png,Normal/masks/Normal-5173.png,True,G14244,train,b64c629ac7fe4696d0bd722e85d73259,d568035a0d6e79ec
Normal-5174.png,Normal,Normal/images/Normal-5174.png,Normal/masks/Normal-5174.png,True,G14245,train,f14b4943e77fa80d8513ebb92a6d3778,9d2ac30f4c454f57
Normal-5175.png,Normal,Normal/images/Normal-5175.png,Normal/masks/Normal-5175.png,True,G14246,train,6ed1679957ee3de3d6ab2ee0c5d68e66,dc2cd30b0f6a2d3c
Normal-5176.png,Normal,Normal/images/Normal-5176.png,Normal/masks/Normal-5176.png,True,G14247,train,45653a76ff90a8f812b674aac75d804a,d13f06461b185f4f
Normal-5177.png,Normal,Normal/images/Normal-5177.png,Normal/masks/Normal-5177.png,True,G14248,train,fc2c3dd3e4a716c8402713bc870d3744,893b161e7b48eb31
Normal-5178.png,Normal,Normal/images/Normal-5178.png,Normal/masks/Normal-5178.png,True,G14249,train,243878a09038338be13707182b7b721a,8168575f1f0f2b38
Normal-5179.png,Normal,Normal/images/Normal-5179.png,Normal/masks/Normal-5179.png,True,G14250,train,6c1171089bfd93ce36383f5e9eb56491,dd60535b411e4d1f
Normal-518.png,Normal,Normal/images/Normal-518.png,Normal/masks/Normal-518.png,True,G14251,train,17b381d80c41fc7644900cefa77f7558,812bc27ada7a78c3
Normal-5180.png,Normal,Normal/images/Normal-5180.png,Normal/masks/Normal-5180.png,True,G14252,train,94abb1a31de27d0309d13a9ffc78bf73,d96e061b4f386c6c
Normal-5181.png,Normal,Normal/images/Normal-5181.png,Normal/masks/Normal-5181.png,True,G14253,val,1eaedeb662861f6858fe42387a5f2292,cc3d430b1dee2d38
Normal-5182.png,Normal,Normal/images/Normal-5182.png,Normal/masks/Normal-5182.png,True,G14254,test,ee64b4440f8b2117e52ceefd70e1120a,fd70530f441f580f
Normal-5183.png,Normal,Normal/images/Normal-5183.png,Normal/masks/Normal-5183.png,True,G14255,test,89d78a2343f493ef62db0f2fdc8d308d,8179061e195f1d5f
Normal-5184.png,Normal,Normal/images/Normal-5184.png,Normal/masks/Normal-5184.png,True,G14256,train,16b3d625d5d3f42ad5127cfb81f1cb2d,897906961a79c1fb
Normal-5185.png,Normal,Normal/images/Normal-5185.png,Normal/masks/Normal-5185.png,True,G14257,val,06eff84bfdce2ebf68d9d20f116519fc,de218d5e384aed68
Normal-5186.png,Normal,Normal/images/Normal-5186.png,Normal/masks/Normal-5186.png,True,G14258,test,f38bd08d79dd6f4166680cc60e07207e,d50a5a0c2f33b7b2
Normal-5187.png,Normal,Normal/images/Normal-5187.png,Normal/masks/Normal-5187.png,True,G14259,train,9709e6d8987b498b04eab1500f4b0263,986d871e494d5af8
Normal-5188.png,Normal,Normal/images/Normal-5188.png,Normal/masks/Normal-5188.png,True,G14260,train,c34958ce3edd9047db7ab0fe36f22a61,8919969e18adcbba
Normal-5189.png,Normal,Normal/images/Normal-5189.png,Normal/masks/Normal-5189.png,True,G14261,val,ee42c84ff8d8d3bf22aed7b11d6b6756,817b061e1b5e5a7a
Normal-519.png,Normal,Normal/images/Normal-519.png,Normal/masks/Normal-519.png,True,G14262,train,2f82c448aec38afcd9f4b76fb3aa9161,913dc61e98c939d9
Normal-5190.png,Normal,Normal/images/Normal-5190.png,Normal/masks/Normal-5190.png,True,G14263,train,198566cf14a98953edab913e1d2c52ef,816807d65a5fa93b
Normal-5191.png,Normal,Normal/images/Normal-5191.png,Normal/masks/Normal-5191.png,True,G14264,train,8b06e6f263980b71e1f339f885e22d02,d43cc3cb5bc44f18
Normal-5192.png,Normal,Normal/images/Normal-5192.png,Normal/masks/Normal-5192.png,True,G14265,train,ec479a1ef4a51c0e07e26376f174fd20,897d065a5bc57869
Normal-5193.png,Normal,Normal/images/Normal-5193.png,Normal/masks/Normal-5193.png,True,G14266,train,0a1a9fd569591eb3afb8ae8c86d2ebe7,997846131bcdab39
Normal-5194.png,Normal,Normal/images/Normal-5194.png,Normal/masks/Normal-5194.png,True,G14267,val,4026c05be11530d20445b2a72ff6d728,897d061a9d4b3be4
Normal-5195.png,Normal,Normal/images/Normal-5195.png,Normal/masks/Normal-5195.png,True,G14268,train,a16407dab540b76420cad2a66e9ef9af,dd7b564b1f400a1d
Normal-5196.png,Normal,Normal/images/Normal-5196.png,Normal/masks/Normal-5196.png,True,G14269,test,1fc238bf820b4b37418852fac1711178,eab5c196b4624b4b
Normal-5197.png,Normal,Normal/images/Normal-5197.png,Normal/masks/Normal-5197.png,True,G14270,train,c97137f767283de86ad367e80ae39fb9,d1390e5a3979add0
Normal-5198.png,Normal,Normal/images/Normal-5198.png,Normal/masks/Normal-5198.png,True,G14271,train,fd542d6d4c83e3f8f12d63f6780019cf,91e99e1a1668bbb8
Normal-5199.png,Normal,Normal/images/Normal-5199.png,Normal/masks/Normal-5199.png,True,G14272,train,3f51a0a8da5fa7c0b7a8747205ba0615,dc3dc2534b4a2f0e
Normal-52.png,Normal,Normal/images/Normal-52.png,Normal/masks/Normal-52.png,True,G14273,train,d86e5550e53ee999132cd1d41ae08425,876186636bed6568
Normal-520.png,Normal,Normal/images/Normal-520.png,Normal/masks/Normal-520.png,True,G14274,train,d06571953a4ac4b0adbb7c823fdf32ee,913bc21af9313d93
Normal-5200.png,Normal,Normal/images/Normal-5200.png,Normal/masks/Normal-5200.png,True,G14275,train,710848501254f87dc0fbdc966cbbf709,d53ec2531f444d1b
Normal-5201.png,Normal,Normal/images/Normal-5201.png,Normal/masks/Normal-5201.png,True,G14276,val,ffabd13f382edeb59163e7181820415e,996ec61bc3071a4f
Normal-5202.png,Normal,Normal/images/Normal-5202.png,Normal/masks/Normal-5202.png,True,G14277,test,3f91f82d8d7b3eaa24e8667266dce691,887f471e185d1b1d
Normal-5203.png,Normal,Normal/images/Normal-5203.png,Normal/masks/Normal-5203.png,True,G14278,test,f9a43b78e481914f661c43278821b945,8b6984120f1e5bdf
Normal-5204.png,Normal,Normal/images/Normal-5204.png,Normal/masks/Normal-5204.png,True,G14279,train,27a35e28fc70347a73d01ee330d8f5ed,c9b03e8b3c27e670
Normal-5205.png,Normal,Normal/images/Normal-5205.png,Normal/masks/Normal-5205.png,True,G14280,train,bbd13f62694abb09f7f1866c8f47a941,896b07961ed57e90
Normal-5206.png,Normal,Normal/images/Normal-5206.png,Normal/masks/Normal-5206.png,True,G14281,train,258ae10dad6f8cd1d40d0aca0edd86ad,903f47423de06fd4
Normal-5207.png,Normal,Normal/images/Normal-5207.png,Normal/masks/Normal-5207.png,True,G14282,train,dd306e0065d109a124875a8300a148be,d959861a1b0dfab8
Normal-5208.png,Normal,Normal/images/Normal-5208.png,Normal/masks/Normal-5208.png,True,G14283,train,2ada65ce59f8d9fa68b0c74638d6d632,d969469b05c737c1
Normal-5209.png,Normal,Normal/images/Normal-5209.png,Normal/masks/Normal-5209.png,True,G14284,train,2ca9210c8ca3002f283004aace1e1e99,c03d4b4f0d8cabbc
Normal-521.png,Normal,Normal/images/Normal-521.png,Normal/masks/Normal-521.png,True,G14285,train,da0066557264e9b854d9ea6057eb7daa,917bca1ac94b1cce
Normal-5210.png,Normal,Normal/images/Normal-5210.png,Normal/masks/Normal-5210.png,True,G14286,train,5424d89218968078d45b14e9fef31852,95280f0e1b795e7a
Normal-5211.png,Normal,Normal/images/Normal-5211.png,Normal/masks/Normal-5211.png,True,G14287,train,83e4d0377be802b451d8f3a59dc607c0,d979861e194d1e17
Normal-5212.png,Normal,Normal/images/Normal-5212.png,Normal/masks/Normal-5212.png,True,G14288,train,8b09fa6825f38595d4c70a466a15ad34,c93906461a5d6d7b
Normal-5213.png,Normal,Normal/images/Normal-5213.png,Normal/masks/Normal-5213.png,True,G14289,train,3788ac7af8f81e6fac9c028f318aeb73,dc2ad30b4c4f1c4f
Normal-5214.png,Normal,Normal/images/Normal-5214.png,Normal/masks/Normal-5214.png,True,G14290,train,f9f808266bd06b085e7272374477de1b,d978c61b19471b39
Normal-5215.png,Normal,Normal/images/Normal-5215.png,Normal/masks/Normal-5215.png,True,G14291,train,c2430f1098d42e5a96a6328f955f4dbd,dc2d42431f1e647d
Normal-5216.png,Normal,Normal/images/Normal-5216.png,Normal/masks/Normal-5216.png,True,G14292,val,5986a7a0c2dbc87373b5f782ffd639b1,8978171a1a577b3a
Normal-5217.png,Normal,Normal/images/Normal-5217.png,Normal/masks/Normal-5217.png,True,G14293,val,52a5eef412e9234b070b429f8b7808ec,dc28d30f0c2d1cbf
Normal-5218.png,Normal,Normal/images/Normal-5218.png,Normal/masks/Normal-5218.png,True,G14294,val,f73f8e2bff42ffc651d1c1e275b905a4,d02dc25353554f1f
Normal-5219.png,Normal,Normal/images/Normal-5219.png,Normal/masks/Normal-5219.png,True,G14295,train,accad4bb8d2233272bd44133bbbdd9e1,dc3c53c35f4c421d
Normal-522.png,Normal,Normal/images/Normal-522.png,Normal/masks/Normal-522.png,True,G14296,test,80f88b361c3e21e336100d069b8abb7a,957eef1b890d0887
Normal-5220.png,Normal,Normal/images/Normal-5220.png,Normal/masks/Normal-5220.png,True,G14297,test,17c9fc36fb7d7cf8ee3029aa9bf2ecd5,cc34c74f19c62cbc
Normal-5221.png,Normal,Normal/images/Normal-5221.png,Normal/masks/Normal-5221.png,True,G14298,train,fcfbead43dee28c613d3a46469f67d5f,897bc71a15455b59
Normal-5222.png,Normal,Normal/images/Normal-5222.png,Normal/masks/Normal-5222.png,True,G14299,test,9b5f30ca412a4a5473651114dbf6936c,d92bc61f480b3de2
Normal-5223.png,Normal,Normal/images/Normal-5223.png,Normal/masks/Normal-5223.png,True,G14300,train,33379dd698a930a5ea1b0a87a72b4ece,d13906c61e5e1acf
Normal-5224.png,Normal,Normal/images/Normal-5224.png,Normal/masks/Normal-5224.png,True,G14301,train,943b83b501571f06dea84b784174a481,dd2cd0d30f4d4c1b
Normal-5225.png,Normal,Normal/images/Normal-5225.png,Normal/masks/Normal-5225.png,True,G14302,val,8f37e40aab526235114c7c1dbc13fb17,997f0619574e585c
Normal-5226.png,Normal,Normal/images/Normal-5226.png,Normal/masks/Normal-5226.png,True,G14303,val,d3aee3b7315ba549edba9bc9541608e9,b86fc61a415f184f
Normal-5227.png,Normal,Normal/images/Normal-5227.png,Normal/masks/Normal-5227.png,True,G14304,test,a8bf9685e0c9c49a4f0982125112c0c0,d97a5358485b5c1e
Normal-5228.png,Normal,Normal/images/Normal-5228.png,Normal/masks/Normal-5228.png,True,G14305,train,f68a787b31917c36b9a5fd4a4e93f422,886cc5561bcf465e
Normal-5229.png,Normal,Normal/images/Normal-5229.png,Normal/masks/Normal-5229.png,True,G14306,test,c3a8a1b2d5c236af12ceb7fd1c325ee8,8129ce86f9e0e5d3
Normal-523.png,Normal,Normal/images/Normal-523.png,Normal/masks/Normal-523.png,True,G14307,train,63c2bcf100fd7b896e725925a490a314,917bca1ad85a1e96
Normal-5230.png,Normal,Normal/images/Normal-5230.png,Normal/masks/Normal-5230.png,True,G14308,train,70112dc116982300df196fa4fea9747c,a80d871a0f0d5f6f
Normal-5231.png,Normal,Normal/images/Normal-5231.png,Normal/masks/Normal-5231.png,True,G14309,train,28cac16864225df678ef4ac5654a29b2,c83c4f0f1b1ae92d
Normal-5232.png,Normal,Normal/images/Normal-5232.png,Normal/masks/Normal-5232.png,True,G14310,train,f3f66294a18f67b42abbc65d26f7de0b,897987161a1d5b79
Normal-5233.png,Normal,Normal/images/Normal-5233.png,Normal/masks/Normal-5233.png,True,G14311,train,1474ed4547959d6e8ce319419da9e789,dc2dd2431f0a0f1f
Normal-5234.png,Normal,Normal/images/Normal-5234.png,Normal/masks/Normal-5234.png,True,G14312,train,75a0975f351a18d9234dab14b5f9db25,805f14185b7f5a5e
Normal-5235.png,Normal,Normal/images/Normal-5235.png,Normal/masks/Normal-5235.png,True,G14313,train,2367768e28e8b4e58c046ff256d78bd6,893987161ecd1f39
Normal-5236.png,Normal,Normal/images/Normal-5236.png,Normal/masks/Normal-5236.png,True,G14314,train,aaa6a789f0e5728ce4fd41256c287c80,817956185b5f7839
Normal-5237.png,Normal,Normal/images/Normal-5237.png,Normal/masks/Normal-5237.png,True,G14315,train,287cb89e2e13ce5ca1bb4399d5409214,dc7c431b19472f98
Normal-5238.png,Normal,Normal/images/Normal-5238.png,Normal/masks/Normal-5238.png,True,G14316,train,7a6cd58b0022a57bc9bd73a0a1a322ac,995f061e191a5e5e
Normal-5239.png,Normal,Normal/images/Normal-5239.png,Normal/masks/Normal-5239.png,True,G14317,train,d1d870dc2a456e8cf82943a385bbd1b6,816996961c5e797a
Normal-524.png,Normal,Normal/images/Normal-524.png,Normal/masks/Normal-524.png,True,G14318,train,8c7153bebc651c8efd36f4b57b239812,9169c75ac94959cd
Normal-5240.png,Normal,Normal/images/Normal-5240.png,Normal/masks/Normal-5240.png,True,G14319,train,6812999dfdbe22c6e84c237f3f0dcbae,817857171a1f564f
Normal-5241.png,Normal,Normal/images/Normal-5241.png,Normal/masks/Normal-5241.png,True,G14320,val,87f4afb9b32096ce66659afcfcb428f4,cc3dc20b4f4f3d82
Normal-5242.png,Normal,Normal/images/Normal-5242.png,Normal/masks/Normal-5242.png,True,G14321,train,1affeb31c4a2443d121ce16667216a42,d43f02575b414f0f
Normal-5243.png,Normal,Normal/images/Normal-5243.png,Normal/masks/Normal-5243.png,True,G14322,test,f17169211dd1f6145c84ced9ef19cfc6,cc3d434f0c2f2db4
Normal-5244.png,Normal,Normal/images/Normal-5244.png,Normal/masks/Normal-5244.png,True,G14323,train,0cbf22df0017678642b2dd59b31a0429,d53f02131d4f5d58
Normal-5245.png,Normal,Normal/images/Normal-5245.png,Normal/masks/Normal-5245.png,True,G14324,train,b9bfa7fcc2cd718288a456d595ad0de4,997f161a495958cb
Normal-5246.png,Normal,Normal/images/Normal-5246.png,Normal/masks/Normal-5246.png,True,G14325,train,3a09d4cd3d5fa494bff5d413a71514e0,ddad32d20bd22bc2
Normal-5247.png,Normal,Normal/images/Normal-5247.png,Normal/masks/Normal-5247.png,True,G14326,train,0e2a3492bac596906f04d147e4063f47,813b161e5f5be843
Normal-5248.png,Normal,Normal/images/Normal-5248.png,Normal/masks/Normal-5248.png,True,G14327,train,e0b92d5689fdfcacda9c01d44d6f9cc6,8920075ad77af9e8
Normal-5249.png,Normal,Normal/images/Normal-5249.png,Normal/masks/Normal-5249.png,True,G14328,train,afbfc75ae3c349c9806809e1d435455e,9c2ad30f0e7c2daa
Normal-525.png,Normal,Normal/images/Normal-525.png,Normal/masks/Normal-525.png,True,G14329,train,cf4f115f69bffd134eb641b38fe7b126,d42bc34bdd49293c
Normal-5250.png,Normal,Normal/images/Normal-5250.png,Normal/masks/Normal-5250.png,True,G14330,test,f9242f1d5a49c73f3ab6beb434534e7f,9168465a3b4ef939
Normal-5251.png,Normal,Normal/images/Normal-5251.png,Normal/masks/Normal-5251.png,True,G14331,train,04134c5d643491427909d0071aafbba9,817f069a1f683a79
Normal-5252.png,Normal,Normal/images/Normal-5252.png,Normal/masks/Normal-5252.png,True,G14332,train,892c210a7905ed066e7a2af86b3aca77,b969861ec71e184f
Normal-5253.png,Normal,Normal/images/Normal-5253.png,Normal/masks/Normal-5253.png,True,G14333,train,775f16c2be375de18b7373b3da1a2693,9969c61a1f5e591a
Normal-5254.png,Normal,Normal/images/Normal-5254.png,Normal/masks/Normal-5254.png,True,G14334,train,ee4d5c16fcc5d542612c853a19ae9369,d42f52574f0d2e29
Normal-5255.png,Normal,Normal/images/Normal-5255.png,Normal/masks/Normal-5255.png,True,G14335,val,d05f80615649428b0f434f96f4f3c298,9979461a194f6b39
Normal-5256.png,Normal,Normal/images/Normal-5256.png,Normal/masks/Normal-5256.png,True,G14336,val,dad07997c8fa906ad00b36f6682250fb,d4fc0702594e6c7d
Normal-5257.png,Normal,Normal/images/Normal-5257.png,Normal/masks/Normal-5257.png,True,G14337,val,61ba96d5e3b8c784f03b987b8a5a9238,d969520f0a1b2f3d
Normal-5258.png,Normal,Normal/images/Normal-5258.png,Normal/masks/Normal-5258.png,True,G14338,test,b26149ec823cbe5a263614b05ef3f9de,d979841e186d1b3b
Normal-5259.png,Normal,Normal/images/Normal-5259.png,Normal/masks/Normal-5259.png,True,G14339,test,b3473cca738251792b923d532a381d85,817bd61a19597b51
Normal-526.png,Normal,Normal/images/Normal-526.png,Normal/masks/Normal-526.png,True,G14340,train,68b7ff49d1b91113c31246c2ab0a6f03,957e0e5f141c3897
Normal-5260.png,Normal,Normal/images/Normal-5260.png,Normal/masks/Normal-5260.png,True,G14341,train,fe50e0be09511da23714a1737fe102bc,897d86161a5f5f18
Normal-5261.png,Normal,Normal/images/Normal-5261.png,Normal/masks/Normal-5261.png,True,G14342,train,65b285d05ea3dbd02e743c7a97b9cb6a,ac6ec14d4d4a5f43
Normal-5262.png,Normal,Normal/images/Normal-5262.png,Normal/masks/Normal-5262.png,True,G14343,train,017e6203692e71a89f79903da37fb010,997b861a5a2b39b8
Normal-5263.png,Normal,Normal/images/Normal-5263.png,Normal/masks/Normal-5263.png,True,G14344,train,d0911b63fc313905fb2c64678cb88e3b,887b17961f9879a2
Normal-5264.png,Normal,Normal/images/Normal-5264.png,Normal/masks/Normal-5264.png,True,G14345,train,fced0ecdcaae59e2b121d87381dc1b70,d93d47461a954e1b
Normal-5265.png,Normal,Normal/images/Normal-5265.png,Normal/masks/Normal-5265.png,True,G14346,val,529018bc704e06b9141c043a44219c4f,815d061a5d7a796e
Normal-5266.png,Normal,Normal/images/Normal-5266.png,Normal/masks/Normal-5266.png,True,G14347,train,20fb4bb6f4ec518468ebbe88dcc0b2ef,bd3dc21b1d0e1b1c
Normal-5267.png,Normal,Normal/images/Normal-5267.png,Normal/masks/Normal-5267.png,True,G14348,train,c6bccf8049f8eeb083549dc6268f2695,9978071e1f555a78
Normal-5268.png,Normal,Normal/images/Normal-5268.png,Normal/masks/Normal-5268.png,True,G14349,test,8028210c0ee7adc85b0efdb34cdbe66c,dd2b56965a49691c
Normal-5269.png,Normal,Normal/images/Normal-5269.png,Normal/masks/Normal-5269.png,True,G14350,train,61c5ecf16a70fa370ef4548d6af9303f,d97947c61a054b1f
Normal-527.png,Normal,Normal/images/Normal-527.png,Normal/masks/Normal-527.png,True,G14351,train,468197dcfee07a31df894da4be52d556,d434d23dfc2c3c4c
Normal-5270.png,Normal,Normal/images/Normal-5270.png,Normal/masks/Normal-5270.png,True,G14352,train,d6df422cd2716d7d4b50eac58584ca42,997d461b160f1a4f
Normal-5271.png,Normal,Normal/images/Normal-5271.png,Normal/masks/Normal-5271.png,True,G14353,train,d039a2e2545a01225552e2b574e92b50,9969061a5f3ae9e8
Normal-5272.png,Normal,Normal/images/Normal-5272.png,Normal/masks/Normal-5272.png,True,G14354,train,2fadc1e0991212059908cba41d64d9b1,dc28d2c34b5fca43
Normal-5273.png,Normal,Normal/images/Normal-5273.png,Normal/masks/Normal-5273.png,True,G14355,train,7187bb567b400eafd02d2ee235398e4f,d93d07061e59693e
Normal-5274.png,Normal,Normal/images/Normal-5274.png,Normal/masks/Normal-5274.png,True,G14356,test,493881066a51f33c8e484ac7fbe01182,c82ac64a7d687b5c
Normal-5275.png,Normal,Normal/images/Normal-5275.png,Normal/masks/Normal-5275.png,True,G14357,test,a60d1f64d9f75c4a083fb43a535dfd5d,dd2fc20d542b78c3
Normal-5276.png,Normal,Normal/images/Normal-5276.png,Normal/masks/Normal-5276.png,True,G14358,val,da8b777a60cf983b44197880364753f7,80ac27c37ac73a9d
Normal-5277.png,Normal,Normal/images/Normal-5277.png,Normal/masks/Normal-5277.png,True,G14359,train,04b9d96c7047b6cba99958d3fb15c938,d53f06531f0d4b1c
Normal-5278.png,Normal,Normal/images/Normal-5278.png,Normal/masks/Normal-5278.png,True,G14360,val,fd7e2537d2020a8f4157276c0c6522d2,9c7cc74a0a8cabab
Normal-5279.png,Normal,Normal/images/Normal-5279.png,Normal/masks/Normal-5279.png,True,G14361,train,078b538c8cfad3a842b72341117e5a2d,dc3dc6c31b15ed08
Normal-528.png,Normal,Normal/images/Normal-528.png,Normal/masks/Normal-528.png,True,G14362,train,73ddcd32c6457a8644c625986e3364fc,d42c833bad3c3ce4
Normal-5280.png,Normal,Normal/images/Normal-5280.png,Normal/masks/Normal-5280.png,True,G14363,train,c4bd23e58976e3a6e81c16a8ad6f375f,cc3d434b1dbcb099
Normal-5281.png,Normal,Normal/images/Normal-5281.png,Normal/masks/Normal-5281.png,True,G14364,train,f46ada984b02ac9824aa0654da75cfac,85683f4b1b38cee4
Normal-5282.png,Normal,Normal/images/Normal-5282.png,Normal/masks/Normal-5282.png,True,G14365,train,b12d15c5ec8ed722bf7ccb7663013dfa,817b061e195d5d5b
Normal-5283.png,Normal,Normal/images/Normal-5283.png,Normal/masks/Normal-5283.png,True,G14366,train,1052aca732124cdd8117b9f50575a594,882ec71b146f3df0
Normal-5284.png,Normal,Normal/images/Normal-5284.png,Normal/masks/Normal-5284.png,True,G14367,train,5b20a7ba73edcfaf725cf5150dc66e58,dd3dc211098e2bbd
Normal-5285.png,Normal,Normal/images/Normal-5285.png,Normal/masks/Normal-5285.png,True,G14368,test,efee29892bcad5dd4dda0595ae20c7e6,99f1669a5d471839
Normal-5286.png,Normal,Normal/images/Normal-5286.png,Normal/masks/Normal-5286.png,True,G14369,train,d8b04782622f5a8fd2a0f8cea0ab505e,dc29d34b5a474b58
Normal-5287.png,Normal,Normal/images/Normal-5287.png,Normal/masks/Normal-5287.png,True,G14370,train,556a9acf540ac3a2ea8d4a3a62d84fec,c83d431e5c1f5a4b
Normal-5288.png,Normal,Normal/images/Normal-5288.png,Normal/masks/Normal-5288.png,True,G14371,val,1b9cbec2c9752bba53bd63bd27c1aa96,987b0516165f1ade
Normal-5289.png,Normal,Normal/images/Normal-5289.png,Normal/masks/Normal-5289.png,True,G14372,train,3ca863c7ccabdedc6d4a840033404c9c,987cc65bcf4a0c4e
Normal-529.png,Normal,Normal/images/Normal-529.png,Normal/masks/Normal-529.png,True,G14373,train,a31673234a497223da8e578c7fca0b0d,902bd25af87a3953
Normal-5290.png,Normal,Normal/images/Normal-5290.png,Normal/masks/Normal-5290.png,True,G14374,val,ac80daae2baccb7fa53613e748da2372,817b571612175f5a
Normal-5291.png,Normal,Normal/images/Normal-5291.png,Normal/masks/Normal-5291.png,True,G14375,test,b49c24661eb334782128ea523534d57e,d83c87471a4b4f4b
Normal-5292.png,Normal,Normal/images/Normal-5292.png,Normal/masks/Normal-5292.png,True,G14376,train,2eab9ad96dacc04834850fe2e8b73f63,c03dcbcf1c48745d
Normal-5293.png,Normal,Normal/images/Normal-5293.png,Normal/masks/Normal-5293.png,True,G14377,test,47d0ff4370aa24c2ffce5d70b3bc6d0d,897f061e170e1f1b
Normal-5294.png,Normal,Normal/images/Normal-5294.png,Normal/masks/Normal-5294.png,True,G14378,train,0bfee71fc8184664befdf70a4186e74f,d42853031f1c5f5f
Normal-5295.png,Normal,Normal/images/Normal-5295.png,Normal/masks/Normal-5295.png,True,G14379,train,949073e8127a8ce497d29d52b6c41b9a,d93d471e146a3a69
Normal-5296.png,Normal,Normal/images/Normal-5296.png,Normal/masks/Normal-5296.png,True,G14380,val,b43a1a0c21d2dfa64bfa423be595e99d,d42f535b4e4c4e1c
Normal-5297.png,Normal,Normal/images/Normal-5297.png,Normal/masks/Normal-5297.png,True,G14381,train,aa232b46b08cf26da24c2110cd536eb6,d56f46590e2e38e4
Normal-5298.png,Normal,Normal/images/Normal-5298.png,Normal/masks/Normal-5298.png,True,G14382,train,38907468c7728f6b4ce585205f0016db,9969c6140b963afb
Normal-5299.png,Normal,Normal/images/Normal-5299.png,Normal/masks/Normal-5299.png,True,G14383,train,1a950bc381b3b81d97a36d5b6e828e65,d97b571e1a193a18
Normal-53.png,Normal,Normal/images/Normal-53.png,Normal/masks/Normal-53.png,True,G14384,train,4b9102f34e085194e3ef576d7d683e13,887a071a1f4d7a79
Normal-530.png,Normal,Normal/images/Normal-530.png,Normal/masks/Normal-530.png,True,G14385,train,2774ff1eec55c97fc3feab227fb81e17,902bc71aba69393d
Normal-5300.png,Normal,Normal/images/Normal-5300.png,Normal/masks/Normal-5300.png,True,G14386,train,5d5d0dfc57a692466c6b4df3a7a2216f,897904c6da18ddcf
Normal-5301.png,Normal,Normal/images/Normal-5301.png,Normal/masks/Normal-5301.png,True,G14387,val,f2ddf380d476198931521ac6301c6a76,cc6cc2c20b4e8f5f
Normal-5302.png,Normal,Normal/images/Normal-5302.png,Normal/masks/Normal-5302.png,True,G14388,train,dacb64a5d22c38f37baecdcca9c75e0a,897b570a1e4f2a3a
Normal-5303.png,Normal,Normal/images/Normal-5303.png,Normal/masks/Normal-5303.png,True,G14389,train,953d0695932d08300260e90601a32c7f,d52c46c31b45ee4e
Normal-5304.png,Normal,Normal/images/Normal-5304.png,Normal/masks/Normal-5304.png,True,G14390,train,7e8220a91b63e1001f4588e6477e168e,dd2cd1930f420f1f
Normal-5305.png,Normal,Normal/images/Normal-5305.png,Normal/masks/Normal-5305.png,True,G14391,train,e6a813ff9d8829f06d600d261bad6e59,887f471a584d6e17
Normal-5306.png,Normal,Normal/images/Normal-5306.png,Normal/masks/Normal-5306.png,True,G14392,train,13c25a98b8ca9f6160cb18ee4c5283db,dd2ed24b1d4c683c
Normal-5307.png,Normal,Normal/images/Normal-5307.png,Normal/masks/Normal-5307.png,True,G14393,train,a54ac52eae6fa7da6f69f8d5a34458bd,dd2d841f1c4d1b1d
Normal-5308.png,Normal,Normal/images/Normal-5308.png,Normal/masks/Normal-5308.png,True,G14394,train,2e1dd095357680d6af14b4ff72f90eb1,d42e428f0abda5f4
Normal-5309.png,Normal,Normal/images/Normal-5309.png,Normal/masks/Normal-5309.png,True,G14395,train,5a9fb3f51add7cd10042bf99970eabb9,813b06065b1fed3b
Normal-531.png,Normal,Normal/images/Normal-531.png,Normal/masks/Normal-531.png,True,G14396,val,66a8525245e110c3a609e8a9cf7d7b4c,943bc31b9c785cd2
Normal-5310.png,Normal,Normal/images/Normal-5310.png,Normal/masks/Normal-5310.png,True,G14397,val,bdaafb884caeb67504c5f2050eb4c581,91781f5a7be1a885
Normal-5311.png,Normal,Normal/images/Normal-5311.png,Normal/masks/Normal-5311.png,True,G14398,train,a9e14b8ec507ef7035b155d523fe0879,9978c3534b473b19
Normal-5312.png,Normal,Normal/images/Normal-5312.png,Normal/masks/Normal-5312.png,True,G14399,test,d3ad26d802d23771aacc1b966f0886a4,cd2ed4c24b0daa2f
Normal-5313.png,Normal,Normal/images/Normal-5313.png,Normal/masks/Normal-5313.png,True,G14400,train,e95554d7fa340035e15db3612417f363,817c86181f5d7b65
Normal-5314.png,Normal,Normal/images/Normal-5314.png,Normal/masks/Normal-5314.png,True,G14401,test,74aa59c7dc2d35d2ec704c1dfe477790,d42b535f0a4f5a0b
Normal-5315.png,Normal,Normal/images/Normal-5315.png,Normal/masks/Normal-5315.png,True,G14402,test,679cad0c64643b6442260c40cad68c4d,f945c61a4d1e3a4b
Normal-5316.png,Normal,Normal/images/Normal-5316.png,Normal/masks/Normal-5316.png,True,G14403,train,ed96acb279b4607502f720b8978cc638,d03c8f191dee3839
Normal-5317.png,Normal,Normal/images/Normal-5317.png,Normal/masks/Normal-5317.png,True,G14404,train,8018da72d2d08a2dfad801348f129d57,997b471a19852f39
Normal-5318.png,Normal,Normal/images/Normal-5318.png,Normal/masks/Normal-5318.png,True,G14405,train,5483dda4356b740d132c2960ce1ac248,987d46c71e494f58
Normal-5319.png,Normal,Normal/images/Normal-5319.png,Normal/masks/Normal-5319.png,True,G14406,train,9c8ab3cc4aadb725e30263e4da48172c,dc2f42d30f08ff0a
Normal-532.png,Normal,Normal/images/Normal-532.png,Normal/masks/Normal-532.png,True,G14407,train,c2d36bf53ebda7fadc462a72f95b8970,d57e8f1b8d4d180c
Normal-5320.png,Normal,Normal/images/Normal-5320.png,Normal/masks/Normal-5320.png,True,G14408,train,005863938b2402f86dd06e6b46cec410,c03c6b434b0d8f6f
Normal-5321.png,Normal,Normal/images/Normal-5321.png,Normal/masks/Normal-5321.png,True,G14409,train,91614cdee9442116a84e4b780fd2496a,9939061f1e4f1a79
Normal-5322.png,Normal,Normal/images/Normal-5322.png,Normal/masks/Normal-5322.png,True,G14410,train,313d3cb96a9a148f8932a6d1731c2001,99efc619471a454e
Normal-5323.png,Normal,Normal/images/Normal-5323.png,Normal/masks/Normal-5323.png,True,G14411,val,27f91b44f6d5a3751a5a771e28b905b3,817987861b155e5f
Normal-5324.png,Normal,Normal/images/Normal-5324.png,Normal/masks/Normal-5324.png,True,G14412,train,f8f4fa9c208f9d66843a7e51ba0a209a,dd2e40d30b454f1f
Normal-5325.png,Normal,Normal/images/Normal-5325.png,Normal/masks/Normal-5325.png,True,G14413,train,f065b896bd0f2c3cf6eef7c2bc0f6bc9,997b851a19ea79a1
Normal-5326.png,Normal,Normal/images/Normal-5326.png,Normal/masks/Normal-5326.png,True,G14414,train,367258879a0c7ab5ad0bad75ad3de7b4,dd02c30f4f0b4f4b
Normal-5327.png,Normal,Normal/images/Normal-5327.png,Normal/masks/Normal-5327.png,True,G14415,train,e98d86b67862f62adeb1c43094f11ebb,9928061a9f6cf9e9
Normal-5328.png,Normal,Normal/images/Normal-5328.png,Normal/masks/Normal-5328.png,True,G14416,val,8550f6225555cb61203a5e47bb711c4b,912ade58594f1f0e
Normal-5329.png,Normal,Normal/images/Normal-5329.png,Normal/masks/Normal-5329.png,True,G14417,val,3a9e35870af945c9aa5439729807de48,dd1e13430f5c5a5a
Normal-533.png,Normal,Normal/images/Normal-533.png,Normal/masks/Normal-533.png,True,G14418,test,f7eba21abcf8616867d0819bde2805e5,913aca5adc5d18d6
Normal-5330.png,Normal,Normal/images/Normal-5330.png,Normal/masks/Normal-5330.png,True,G14419,test,388c169bb0ecbcbb57a13a8c7921ee09,897d46125f471a5b
Normal-5331.png,Normal,Normal/images/Normal-5331.png,Normal/masks/Normal-5331.png,True,G14420,test,0bc38130d381e49dd0e4ba7d03ccceb7,996d471a484f5ae9
Normal-5332.png,Normal,Normal/images/Normal-5332.png,Normal/masks/Normal-5332.png,True,G14421,train,ecfb214381f09f75a018cef03a5b0c58,996807121f5f38cf
Normal-5333.png,Normal,Normal/images/Normal-5333.png,Normal/masks/Normal-5333.png,True,G14422,train,b3aa0a1cbb978e7af27ffb855e7d9811,9168861a195d7b5f
Normal-5334.png,Normal,Normal/images/Normal-5334.png,Normal/masks/Normal-5334.png,True,G14423,train,a3e89bdbe3f26077e7a487eed83ed3c1,d42e52d34b595743
Normal-5335.png,Normal,Normal/images/Normal-5335.png,Normal/masks/Normal-5335.png,True,G14424,train,422441df71bc14a581dbba599cdd98ca,d13c46431b4f6e39
Normal-5336.png,Normal,Normal/images/Normal-5336.png,Normal/masks/Normal-5336.png,True,G14425,train,a1925b8350d57e5004423436b82a40fe,913d86861d4f971e
Normal-5337.png,Normal,Normal/images/Normal-5337.png,Normal/masks/Normal-5337.png,True,G14426,train,e7cf91f958f6da25861c9c12a1557cb5,897d86c69b190d4f
Normal-5338.png,Normal,Normal/images/Normal-5338.png,Normal/masks/Normal-5338.png,True,G14427,train,1d92178cd440e6635e765f292edaf7a8,807dc61b5d6c3939
Normal-5339.png,Normal,Normal/images/Normal-5339.png,Normal/masks/Normal-5339.png,True,G14428,train,48d738f5c4dc77c5d0d96f898bbfa631,9969c61a114f7b1d
Normal-534.png,Normal,Normal/images/Normal-534.png,Normal/masks/Normal-534.png,True,G14429,train,1c51a613f4b2d6f09ed377809a58b73e,d46ec31fe50c2cc6
Normal-5340.png,Normal,Normal/images/Normal-5340.png,Normal/masks/Normal-5340.png,True,G14430,train,b2553b9809bb575e8e68f7d97d872ec2,997b851e1e451b39
Normal-5341.png,Normal,Normal/images/Normal-5341.png,Normal/masks/Normal-5341.png,True,G14431,train,79d9ea5db9d673b9ae39228ffc56eda7,816887961f575e5a
Normal-5342.png,Normal,Normal/images/Normal-5342.png,Normal/masks/Normal-5342.png,True,G14432,train,d3960dd5987e39dfbd1be2c970dd3749,997d071ad9c938e1
Normal-5343.png,Normal,Normal/images/Normal-5343.png,Normal/masks/Normal-5343.png,True,G14433,train,4331627905af8b30e64367032928f4de,9c0c435b0b4e7e7c
Normal-5344.png,Normal,Normal/images/Normal-5344.png,Normal/masks/Normal-5344.png,True,G14434,train,b6977c98de5605bc0aa5de7b60d7442e,9928165ddbaae9a4
Normal-5345.png,Normal,Normal/images/Normal-5345.png,Normal/masks/Normal-5345.png,True,G14435,val,b731d313f2d75e6190e7c4450145eba7,d43d0a855d2cf4cb
Normal-5346.png,Normal,Normal/images/Normal-5346.png,Normal/masks/Normal-5346.png,True,G14436,val,fef9853fd35e87ed16e2004078acdc97,8818425a1f2d3fbf
Normal-5347.png,Normal,Normal/images/Normal-5347.png,Normal/masks/Normal-5347.png,True,G14437,test,9f35e9962128cfa11aae2d4414696d4d,dd2c46c31a5dcd4a
Normal-5348.png,Normal,Normal/images/Normal-5348.png,Normal/masks/Normal-5348.png,True,G14438,test,8ff54c4959fd4cb3d445f28af45c5814,d938421f5c2b3d66
Normal-5349.png,Normal,Normal/images/Normal-5349.png,Normal/masks/Normal-5349.png,True,G14439,val,5ffa2f8d6cd313d5e5c04c16fb4795c8,817bd61e18477ad1
Normal-535.png,Normal,Normal/images/Normal-535.png,Normal/masks/Normal-535.png,True,G14440,test,60c2b904182c794af2adbe847ff84d8c,803ada5edc559696
Normal-5350.png,Normal,Normal/images/Normal-5350.png,Normal/masks/Normal-5350.png,True,G14441,train,9447fa01c227bffaff9fef5838dbe4c0,8979851e184d5f79
Normal-5351.png,Normal,Normal/images/Normal-5351.png,Normal/masks/Normal-5351.png,True,G14442,train,f607d771a5a4bdc1ad4df0917a761a54,dd2dc21b134d1e39
Normal-5352.png,Normal,Normal/images/Normal-5352.png,Normal/masks/Normal-5352.png,True,G14443,train,39eb77945858fb5704996d775afff57e,b939071e1a495b79
Normal-5353.png,Normal,Normal/images/Normal-5353.png,Normal/masks/Normal-5353.png,True,G14444,train,33f4f43ca412f38ebe088f0616932896,817b061a5be879b9
Normal-5354.png,Normal,Normal/images/Normal-5354.png,Normal/masks/Normal-5354.png,True,G14445,train,e3e92b5a7b53eac54478d3df3d336e16,887d07165a5d4d79
Normal-5355.png,Normal,Normal/images/Normal-5355.png,Normal/masks/Normal-5355.png,True,G14446,val,425ce52e67fe80ee4839698f9109d7ad,d92d471b51471d5c
Normal-5356.png,Normal,Normal/images/Normal-5356.png,Normal/masks/Normal-5356.png,True,G14447,train,7291ad2eb6c3c50a3de255a0b6968c9c,d87985561685571f
Normal-5357.png,Normal,Normal/images/Normal-5357.png,Normal/masks/Normal-5357.png,True,G14448,train,d0883055f18475a037f578f9b9ac8e7e,995f061a5978d2ce
Normal-5358.png,Normal,Normal/images/Normal-5358.png,Normal/masks/Normal-5358.png,True,G14449,test,905b22d978764b00b399f01ea21372a6,d52e524b5f4a0f1c
Normal-5359.png,Normal,Normal/images/Normal-5359.png,Normal/masks/Normal-5359.png,True,G14450,train,529fe8f9f95c69ff8d3007a93fcaa28d,d928d71b192d38e9
Normal-536.png,Normal,Normal/images/Normal-536.png,Normal/masks/Normal-536.png,True,G14451,train,1ffca2bb966cbdc707553eedc5ea5ba7,d12f061a5d5b386d
Normal-5360.png,Normal,Normal/images/Normal-5360.png,Normal/masks/Normal-5360.png,True,G14452,train,13903aea8d493e1122137d5b4075420e,81691e141b5f7979
Normal-5361.png,Normal,Normal/images/Normal-5361.png,Normal/masks/Normal-5361.png,True,G14453,train,13d706dc1233f8a84089506239efa8a4,dc3d434b5c026f3c
Normal-5362.png,Normal,Normal/images/Normal-5362.png,Normal/masks/Normal-5362.png,True,G14454,val,3a1a77d6a89d866b5b607f9cfb9641cf,d42e52574b4d4d1d
Normal-5363.png,Normal,Normal/images/Normal-5363.png,Normal/masks/Normal-5363.png,True,G14455,train,ed2acd8cd503992daf8504967226271d,9c2c43c30b5f4f4e
Normal-5364.png,Normal,Normal/images/Normal-5364.png,Normal/masks/Normal-5364.png,True,G14456,train,20ae7fd0b6af8e807be6df8bf20b363e,fc44430b0e0f3d6f
Normal-5365.png,Normal,Normal/images/Normal-5365.png,Normal/masks/Normal-5365.png,True,G14457,train,76466d00221c57f262c9026e31d92dc4,f879c65e184d5919
Normal-5366.png,Normal,Normal/images/Normal-5366.png,Normal/masks/Normal-5366.png,True,G14458,train,538e495c43fcf68af1f8aa7b27d4acdd,d42bc2ff5c0c6b18
Normal-5367.png,Normal,Normal/images/Normal-5367.png,Normal/masks/Normal-5367.png,True,G14459,test,c40d589d7274e493abbfd7a090597dc9,997f061a1d4b195e
Normal-5368.png,Normal,Normal/images/Normal-5368.png,Normal/masks/Normal-5368.png,True,G14460,val,b0acf4c45b926dae0919b31284b10faf,c635e8282727a7a7
Normal-5369.png,Normal,Normal/images/Normal-5369.png,Normal/masks/Normal-5369.png,True,G14461,test,3ab434cf5d4210e3eeb1a68b2ec56721,c17b56561809aa7f
Normal-537.png,Normal,Normal/images/Normal-537.png,Normal/masks/Normal-537.png,True,G14462,train,be00840a5c0a7a38a057d8d572eaa9dc,902de61ecd491dcd
Normal-5370.png,Normal,Normal/images/Normal-5370.png,Normal/masks/Normal-5370.png,True,G14463,train,570b33b512e814e9b0449e46fa0480cd,fc2cc30b0daa2cbd
Normal-5371.png,Normal,Normal/images/Normal-5371.png,Normal/masks/Normal-5371.png,True,G14464,train,740c2372d9cc6ef06be5f1e8271559b4,817b861a5d4f3b19
Normal-5372.png,Normal,Normal/images/Normal-5372.png,Normal/masks/Normal-5372.png,True,G14465,train,e65cfcad2ba716ef6676998193125a6e,d52e56430f1e4b0f
Normal-5373.png,Normal,Normal/images/Normal-5373.png,Normal/masks/Normal-5373.png,True,G14466,train,a75b77c7e763d7e0f5d91a01a631a8f8,997b87021fc47a39
Normal-5374.png,Normal,Normal/images/Normal-5374.png,Normal/masks/Normal-5374.png,True,G14467,train,e123ec1b30aa220a8451ad7f79bbfad0,9803061e4b5f597f
Normal-5375.png,Normal,Normal/images/Normal-5375.png,Normal/masks/Normal-5375.png,True,G14468,train,56eccd8b7fad4fe82dfeb82866b7a156,897bc61a57471a59
Normal-5376.png,Normal,Normal/images/Normal-5376.png,Normal/masks/Normal-5376.png,True,G14469,train,852b40427bfadb5440e091087709dd49,bc0d871b071f0d4f
Normal-5377.png,Normal,Normal/images/Normal-5377.png,Normal/masks/Normal-5377.png,True,G14470,train,7fe8161cd0ac41fa18510bf7640faaab,b96dc61a194f1c47
Normal-5378.png,Normal,Normal/images/Normal-5378.png,Normal/masks/Normal-5378.png,True,G14471,train,ff863cf44a317fc2f9d554f3dbdbeb61,9968061b9e5e595e
Normal-5379.png,Normal,Normal/images/Normal-5379.png,Normal/masks/Normal-5379.png,True,G14472,train,d03912e78ec1bb543220d3d1895ab1f3,99691e19175e78b8
Normal-538.png,Normal,Normal/images/Normal-538.png,Normal/masks/Normal-538.png,True,G14473,train,004d885b31c0bb9a292af2c1181d89c9,d47cc21b8b2f3874
Normal-5380.png,Normal,Normal/images/Normal-5380.png,Normal/masks/Normal-5380.png,True,G14474,train,58e9c904d8fa3deea8ad31c41afb34da,cd5cd25b0f0c1e1d
Normal-5381.png,Normal,Normal/images/Normal-5381.png,Normal/masks/Normal-5381.png,True,G14475,train,500c406991b3a5d66415c3272c09c3c4,9d6cc0121b4d1f5f
Normal-5382.png,Normal,Normal/images/Normal-5382.png,Normal/masks/Normal-5382.png,True,G14476,val,6da9100ebec51de4b69f67c1f171cf3d,81fd871b15073d1d
Normal-5383.png,Normal,Normal/images/Normal-5383.png,Normal/masks/Normal-5383.png,True,G14477,val,7d89fc0588e5cbd7014fad62d8651abe,817f061a1d5e195f
Normal-5384.png,Normal,Normal/images/Normal-5384.png,Normal/masks/Normal-5384.png,True,G14478,val,c8ac76fbfc49e63b59c934c9da503200,957f06c61e18c9e9
Normal-5385.png,Normal,Normal/images/Normal-5385.png,Normal/masks/Normal-5385.png,True,G14479,train,5486f17a724b6c04289b3699cc197c39,dd2cd35b55425d44
Normal-5386.png,Normal,Normal/images/Normal-5386.png,Normal/masks/Normal-5386.png,True,G14480,test,1b3866f4f70835af0573758fdbfe6379,bc6cc6921b4d071f
Normal-5387.png,Normal,Normal/images/Normal-5387.png,Normal/masks/Normal-5387.png,True,G14481,test,dc5bbfc3b43fd790b7f09e77931b30d3,dd2c93870f540f1d
Normal-5388.png,Normal,Normal/images/Normal-5388.png,Normal/masks/Normal-5388.png,True,G14482,test,3ec1799ed2f0e28f6dfdef7a6da67508,997d84161e4f5a1e
Normal-5389.png,Normal,Normal/images/Normal-5389.png,Normal/masks/Normal-5389.png,True,G14483,train,6a0dade005e936d2296ba3d3bd1c9aff,dd3dd45a1741191d
Normal-539.png,Normal,Normal/images/Normal-539.png,Normal/masks/Normal-539.png,True,G14484,train,a8c3ba44cebacd2a3c320052c4f25178,d06dc71acd4d3cc4
Normal-5390.png,Normal,Normal/images/Normal-5390.png,Normal/masks/Normal-5390.png,True,G14485,train,0f50b365488d1de20cf2268ea6448a3a,81780f163a1f3d3d
Normal-5391.png,Normal,Normal/images/Normal-5391.png,Normal/masks/Normal-5391.png,True,G14486,train,14892fb97685212f345f9157a50d94aa,9b695c1c7ba9e105
Normal-5392.png,Normal,Normal/images/Normal-5392.png,Normal/masks/Normal-5392.png,True,G14487,train,92dd47a0844b45b6cc4785840fdf961b,d97bc71a570b1845
Normal-5393.png,Normal,Normal/images/Normal-5393.png,Normal/masks/Normal-5393.png,True,G14488,train,fa55325a4792e8e931cb999269bca1ec,dc2cd30b55420f6f
Normal-5394.png,Normal,Normal/images/Normal-5394.png,Normal/masks/Normal-5394.png,True,G14489,train,2cf4a94d749f9de74cbc8a0b4561e86e,d93c4619156f39b1
Normal-5395.png,Normal,Normal/images/Normal-5395.png,Normal/masks/Normal-5395.png,True,G14490,val,03477c369e9101b39ad1ab8bcd8f8270,8173121f6b3de325
Normal-5396.png,Normal,Normal/images/Normal-5396.png,Normal/masks/Normal-5396.png,True,G14491,train,44e1ce8d7c0c05b2e534c550f1d9c152,d42cd3430d4dcb4f
Normal-5397.png,Normal,Normal/images/Normal-5397.png,Normal/masks/Normal-5397.png,True,G14492,test,e3695a57a6f92c8b4e5cdf449b3e5993,996d86161b455eb9
Normal-5398.png,Normal,Normal/images/Normal-5398.png,Normal/masks/Normal-5398.png,True,G14493,train,6ebae663ec32d8ab699ebbf9f3bc92ec,990f4618497b38ef
Normal-5399.png,Normal,Normal/images/Normal-5399.png,Normal/masks/Normal-5399.png,True,G14494,train,46e417272929b58b0f8297103dd14d77,9978061c5a5e78f9
Normal-54.png,Normal,Normal/images/Normal-54.png,Normal/masks/Normal-54.png,True,G14495,train,d8c7978c7e0f308934f65d1cfa51f059,dd2c4582531d5f4d
Normal-540.png,Normal,Normal/images/Normal-540.png,Normal/masks/Normal-540.png,True,G14496,train,ebc9c7513ae9fdc353ec959c48f6e75b,d46ec31bcd6c9c48
Normal-5400.png,Normal,Normal/images/Normal-5400.png,Normal/masks/Normal-5400.png,True,G14497,train,737adff87d95d3a78c6c999e2de89eb4,dd2a42031f4d6e3d
Normal-5401.png,Normal,Normal/images/Normal-5401.png,Normal/masks/Normal-5401.png,True,G14498,val,ccbec9d31253427630f0c7a1c3670d5a,987d47165a471e5a
Normal-5402.png,Normal,Normal/images/Normal-5402.png,Normal/masks/Normal-5402.png,True,G14499,val,7cd5ed472ff58382d2ddcced5871c267,807b57961a396bb1
Normal-5403.png,Normal,Normal/images/Normal-5403.png,Normal/masks/Normal-5403.png,True,G14500,test,11df26cea2d3a50d8a10f6b6c95dafe2,dc05871b451e5d5e
Normal-5404.png,Normal,Normal/images/Normal-5404.png,Normal/masks/Normal-5404.png,True,G14501,test,fcf1ff979209b8940caea257b6ddb17b,807f871e1ad71a39
Normal-5405.png,Normal,Normal/images/Normal-5405.png,Normal/masks/Normal-5405.png,True,G14502,train,c621500deaa0cb54b902f201703800c4,83601f1e5b7b782a
Normal-5406.png,Normal,Normal/images/Normal-5406.png,Normal/masks/Normal-5406.png,True,G14503,train,02af2c8b5ec92de7109c6c0177e824f3,897f0618497f195e
Normal-5407.png,Normal,Normal/images/Normal-5407.png,Normal/masks/Normal-5407.png,True,G14504,train,fcdae686901bf155b096bc26da6f2acc,dc2a534b4d4f0a4f
Normal-5408.png,Normal,Normal/images/Normal-5408.png,Normal/masks/Normal-5408.png,True,G14505,train,aba7de2a31b9c7c375af20b37fe8996e,c97dd01f951b1b84
Normal-5409.png,Normal,Normal/images/Normal-5409.png,Normal/masks/Normal-5409.png,True,G14506,train,bb935875c8f05a82fe51c2d8a5f026d9,c87d4a084c7e9f87
Normal-541.png,Normal,Normal/images/Normal-541.png,Normal/masks/Normal-541.png,True,G14507,train,749331b1a978e437c504d99a4b59f46d,912bca18c859ddcf
Normal-5410.png,Normal,Normal/images/Normal-5410.png,Normal/masks/Normal-5410.png,True,G14508,train,da4bd70ea71c7ff53e73643a35faaf7a,f939875a181c4b3b
Normal-5411.png,Normal,Normal/images/Normal-5411.png,Normal/masks/Normal-5411.png,True,G14509,train,9fd6896960871ca2d820ade8420c7adf,a94d061e59ed68f8
Normal-5412.png,Normal,Normal/images/Normal-5412.png,Normal/masks/Normal-5412.png,True,G14510,train,9c44facc1e320d163eae302851f6498f,d909861e511f5d4f
Normal-5413.png,Normal,Normal/images/Normal-5413.png,Normal/masks/Normal-5413.png,True,G14511,train,8de34d76757b36c762ce0ecd3a2fb0f0,9979d6021a075f7a
Normal-5414.png,Normal,Normal/images/Normal-5414.png,Normal/masks/Normal-5414.png,True,G14512,train,b0aad3d7f91d1cf960b1cad334eefff8,dd28c4171b534f1e
Normal-5415.png,Normal,Normal/images/Normal-5415.png,Normal/masks/Normal-5415.png,True,G14513,train,8f6bff189f34651f2219306cdacc7a5d,cc3cc21f904f1f5a
Normal-5416.png,Normal,Normal/images/Normal-5416.png,Normal/masks/Normal-5416.png,True,G14514,val,73982cc1e850fad16ad482a1e0ff1c66,913d06171959ed3b
Normal-5417.png,Normal,Normal/images/Normal-5417.png,Normal/masks/Normal-5417.png,True,G14515,train,91883d70cdbb177754bb6734c8d2796e,996906161f6e7a39
Normal-5418.png,Normal,Normal/images/Normal-5418.png,Normal/masks/Normal-5418.png,True,G14516,test,f0927879125c3f7fefb8f3b74bb39157,8979469e5f891e89
Normal-5419.png,Normal,Normal/images/Normal-5419.png,Normal/masks/Normal-5419.png,True,G14517,train,569e45f420fbd6952e79cdd2f2b94618,943c45c79716d353
Normal-542.png,Normal,Normal/images/Normal-542.png,Normal/masks/Normal-542.png,True,G14518,val,53908d3af6323fedb1e8c27624a34a2d,812bc61ac879b9db
Normal-5420.png,Normal,Normal/images/Normal-5420.png,Normal/masks/Normal-5420.png,True,G14519,val,f1d12adac588c44a8a62dfcd34700cb9,d92b86069ec6bab8
Normal-5421.png,Normal,Normal/images/Normal-5421.png,Normal/masks/Normal-5421.png,True,G14520,test,9004eb2b970a93f00535b3a6e303fe2d,c97d461e4959194f
Normal-5422.png,Normal,Normal/images/Normal-5422.png,Normal/masks/Normal-5422.png,True,G14521,test,71e37127800e4ebb1233922691ba5a4d,813906de5859ed6e
Normal-5423.png,Normal,Normal/images/Normal-5423.png,Normal/masks/Normal-5423.png,True,G14522,train,04fcfe9d6d86f9b66c602a5db39e390e,997bc71e164f1a18
Normal-5424.png,Normal,Normal/images/Normal-5424.png,Normal/masks/Normal-5424.png,True,G14523,train,32e5a58e5a24ac4c24a0041eb3bfbf5a,816946da597b7938
Normal-5425.png,Normal,Normal/images/Normal-5425.png,Normal/masks/Normal-5425.png,True,G14524,train,d52b7e68e8f4cf5ec4c921212da98173,d929178e16187de9
Normal-5426.png,Normal,Normal/images/Normal-5426.png,Normal/masks/Normal-5426.png,True,G14525,train,33800ea5043a091098910e648b5be19f,d43e831d3c563e2c
Normal-5427.png,Normal,Normal/images/Normal-5427.png,Normal/masks/Normal-5427.png,True,G14526,train,6879bc8043f92e3d1c1c8b5113cca855,9969161a175e79e8
Normal-5428.png,Normal,Normal/images/Normal-5428.png,Normal/masks/Normal-5428.png,True,G14527,train,5081f78cd8df8cdd7dade7c259c8efdc,c03d2b934b1d5c4f
Normal-5429.png,Normal,Normal/images/Normal-5429.png,Normal/masks/Normal-5429.png,True,G14528,train,aec83c4ed4378630b97cc2a8abe6e872,d939c71a0ece1a8e
Normal-543.png,Normal,Normal/images/Normal-543.png,Normal/masks/Normal-543.png,True,G14529,train,541a4f656d50d3b03d4a61fcd7579324,d02dc73a6d6c3cc8
Normal-5430.png,Normal,Normal/images/Normal-5430.png,Normal/masks/Normal-5430.png,True,G14530,train,1326c2ed63de63d41ad3297ca796e936,c93d065a186d7979
Normal-5431.png,Normal,Normal/images/Normal-5431.png,Normal/masks/Normal-5431.png,True,G14531,train,bfa40c33ccbdf76478d8adc52cb46bdd,c169565e0f2f3d24
Normal-5432.png,Normal,Normal/images/Normal-5432.png,Normal/masks/Normal-5432.png,True,G14532,val,387f13fb9ed7fed538f7867f12864eed,dc2d420f086f3ceb
Normal-5433.png,Normal,Normal/images/Normal-5433.png,Normal/masks/Normal-5433.png,True,G14533,train,4702d2fb22b31f2ffbfead0b08c44d2d,da2f840b2f36b5b0
Normal-5434.png,Normal,Normal/images/Normal-5434.png,Normal/masks/Normal-5434.png,True,G14534,train,6ae7d4b4b1a7166598b024705c8f8c03,dd7d4607591a8749
Normal-5435.png,Normal,Normal/images/Normal-5435.png,Normal/masks/Normal-5435.png,True,G14535,train,07f7b73b7602630fb63b981c58da4141,883d065a495e78ef
Normal-5436.png,Normal,Normal/images/Normal-5436.png,Normal/masks/Normal-5436.png,True,G14536,train,8d649db7aeb4f661b99f3f9160abccf2,9925075a055e577b
Normal-5437.png,Normal,Normal/images/Normal-5437.png,Normal/masks/Normal-5437.png,True,G14537,val,f84f5f22d9db8c41a00b60f7392688c5,d42f435b4a4f1b0b
Normal-5438.png,Normal,Normal/images/Normal-5438.png,Normal/masks/Normal-5438.png,True,G14538,val,eaabc0202654b344e18ee8a243b801b8,890d8e5e5879eee0
Normal-5439.png,Normal,Normal/images/Normal-5439.png,Normal/masks/Normal-5439.png,True,G14539,test,c791bc20af5c2eac82976699a604433a,d03dc20f9a4f1f38
Normal-544.png,Normal,Normal/images/Normal-544.png,Normal/masks/Normal-544.png,True,G14540,test,7d7654fc49a647ef85a17bf3cac4320a,912feb3b4b4d1c44
Normal-5440.png,Normal,Normal/images/Normal-5440.png,Normal/masks/Normal-5440.png,True,G14541,test,b01326e587bd1d6fc59cf112943252f5,8179a6479d0aee2b
Normal-5441.png,Normal,Normal/images/Normal-5441.png,Normal/masks/Normal-5441.png,True,G14542,train,d7afbe84cca83f00d7b5f3d8c219ce15,9979065e1a496af9
Normal-5442.png,Normal,Normal/images/Normal-5442.png,Normal/masks/Normal-5442.png,True,G14543,train,fc9ab14f0ddcdcb12a026fd7cf30c02c,cc3d52530b0fcb0f
Normal-5443.png,Normal,Normal/images/Normal-5443.png,Normal/masks/Normal-5443.png,True,G14544,train,ea212b55148d4333bda11cff1a0626fb,dd2d40c31b1f4d4d
Normal-5444.png,Normal,Normal/images/Normal-5444.png,Normal/masks/Normal-5444.png,True,G14545,train,66c843969fa718c99902fe8cf499ba36,b97906d61a1d0d3b
Normal-5445.png,Normal,Normal/images/Normal-5445.png,Normal/masks/Normal-5445.png,True,G14546,train,439d86013dffc4ae0c62c0f4853e8faf,913a061a197d7b79
Normal-5446.png,Normal,Normal/images/Normal-5446.png,Normal/masks/Normal-5446.png,True,G14547,train,c563704ff1f2035939dcacddf6de64cf,dd2e424b0f544f5c
Normal-5447.png,Normal,Normal/images/Normal-5447.png,Normal/masks/Normal-5447.png,True,G14548,train,e4f090702cc8b13465a9d177c94963c3,d52d025b0b4e5f0f
Normal-5448.png,Normal,Normal/images/Normal-5448.png,Normal/masks/Normal-5448.png,True,G14549,test,5793c86000a5b07f44df9c0a21402f58,dc2ed0cb0bc30b2f
Normal-5449.png,Normal,Normal/images/Normal-5449.png,Normal/masks/Normal-5449.png,True,G14550,val,92fdd457842edb9ab858a78aae8855ad,ccbcc34d52471c5e
Normal-545.png,Normal,Normal/images/Normal-545.png,Normal/masks/Normal-545.png,True,G14551,train,75f847dd15248b4c02380b5f92dbdef2,91adce3a586d3c70
Normal-5450.png,Normal,Normal/images/Normal-5450.png,Normal/masks/Normal-5450.png,True,G14552,train,fad622d8be009db53cae229e8480817a,d97dc61a19453a39
Normal-5451.png,Normal,Normal/images/Normal-5451.png,Normal/masks/Normal-5451.png,True,G14553,train,36a37f97192777c924d276af07ed7d73,817fc60e19ee44ba
Normal-5452.png,Normal,Normal/images/Normal-5452.png,Normal/masks/Normal-5452.png,True,G14554,train,f58d5f29ce16dd636bca3933aba4482d,987d861a451fa94f
Normal-5453.png,Normal,Normal/images/Normal-5453.png,Normal/masks/Normal-5453.png,True,G14555,train,99867fa80b4b232ae8d3dc0eb4a21c08,d43f425759424f1d
Normal-5454.png,Normal,Normal/images/Normal-5454.png,Normal/masks/Normal-5454.png,True,G14556,train,1c94ec31a43d8817855a44d539ed1c6b,806a875e5f5a195e
Normal-5455.png,Normal,Normal/images/Normal-5455.png,Normal/masks/Normal-5455.png,True,G14557,train,98d48d1b26aa5a14e714fd15e4afea15,d92844579f52c70f
Normal-5456.png,Normal,Normal/images/Normal-5456.png,Normal/masks/Normal-5456.png,True,G14558,val,a8ade6ec33dba75ad2e2582f75cf0bf4,8168879e1f0f1d1f
Normal-5457.png,Normal,Normal/images/Normal-5457.png,Normal/masks/Normal-5457.png,True,G14559,test,55edbdb4ba779568cd78fde86719d757,d93e471a194b1cad
Normal-5458.png,Normal,Normal/images/Normal-5458.png,Normal/masks/Normal-5458.png,True,G14560,val,eb4927ddea9c32ba57912698efd4e31c,8179560a181f6f7b
Normal-5459.png,Normal,Normal/images/Normal-5459.png,Normal/masks/Normal-5459.png,True,G14561,test,a69322527693ad76d0d9558c86d5e9df,9978071e1f4d7a38
Normal-546.png,Normal,Normal/images/Normal-546.png,Normal/masks/Normal-546.png,True,G14562,test,faa47eb36b703aa591c4ba8cbe2c8cf6,d462c77fc50d6087
Normal-5460.png,Normal,Normal/images/Normal-5460.png,Normal/masks/Normal-5460.png,True,G14563,train,b12ba223006e13e233ab0c7816ac3f90,887d8616c757115f
Normal-5461.png,Normal,Normal/images/Normal-5461.png,Normal/masks/Normal-5461.png,True,G14564,train,ef69fc81169ec54ed9dd415123d34136,dc0e870d4c7a70f3
Normal-5462.png,Normal,Normal/images/Normal-5462.png,Normal/masks/Normal-5462.png,True,G14565,train,7932fd3b819a043993859f5bbf5be740,dd2d821b0d4e1bd5
Normal-5463.png,Normal,Normal/images/Normal-5463.png,Normal/masks/Normal-5463.png,True,G14566,train,c853776ba2e5a7a5dda734ffd422a405,dd28d25f564a1c5a
Normal-5464.png,Normal,Normal/images/Normal-5464.png,Normal/masks/Normal-5464.png,True,G14567,train,56949f01d0312555d9e18589685031a8,c93d56c21b1eca2b
Normal-5465.png,Normal,Normal/images/Normal-5465.png,Normal/masks/Normal-5465.png,True,G14568,train,802abd0883ec14745d10fa61308e9b75,dd2f82525f452f18
Normal-5466.png,Normal,Normal/images/Normal-5466.png,Normal/masks/Normal-5466.png,True,G14569,test,19dd5dd5b296d5464e0d4dabfe448ef7,9c6e40521b455f5f
Normal-5467.png,Normal,Normal/images/Normal-5467.png,Normal/masks/Normal-5467.png,True,G14570,train,9bfac45fad2a1555b77dc1627716d61b,cd3d420f9a693de0
Normal-5468.png,Normal,Normal/images/Normal-5468.png,Normal/masks/Normal-5468.png,True,G14571,train,26b1e3ae2d0e168bee7d0de9665d1986,d139425f585b0b5b
Normal-5469.png,Normal,Normal/images/Normal-5469.png,Normal/masks/Normal-5469.png,True,G14572,train,dda63d5bf19a76f5b8e555210fae15b8,dd38d35353c24754
Normal-547.png,Normal,Normal/images/Normal-547.png,Normal/masks/Normal-547.png,True,G14573,train,b4da0cb9a52b92ca22db40bfc577eada,d4eeab5bc849b8c0
Normal-5470.png,Normal,Normal/images/Normal-5470.png,Normal/masks/Normal-5470.png,True,G14574,train,3e26d8b7557170b6510289f4a4ab01f4,dd3a421b086dbea9
Normal-5471.png,Normal,Normal/images/Normal-5471.png,Normal/masks/Normal-5471.png,True,G14575,train,b677f21c7313a40419f38d9e3e752bb4,d46f47490d4c7c4b
Normal-5472.png,Normal,Normal/images/Normal-5472.png,Normal/masks/Normal-5472.png,True,G14576,train,e19d249768249bde9a5c0c8374137e4f,b92dc71e00c51fd9
Normal-5473.png,Normal,Normal/images/Normal-5473.png,Normal/masks/Normal-5473.png,True,G14577,train,78d4a18d1523fbe658a683e2e401256e,d13b45875e1a9c56
Normal-5474.png,Normal,Normal/images/Normal-5474.png,Normal/masks/Normal-5474.png,True,G14578,val,c52e6ba8daccfea37b22a1e93a8be12c,997d075e18497d38
Normal-5475.png,Normal,Normal/images/Normal-5475.png,Normal/masks/Normal-5475.png,True,G14579,test,708bff08feb9b2279193cbc4cbcaac23,c03d420d1f4e6d3f
Normal-5476.png,Normal,Normal/images/Normal-5476.png,Normal/masks/Normal-5476.png,True,G14580,train,178f086760f9070aacc0bb6624a547bf,997d849f13855f18
Normal-5477.png,Normal,Normal/images/Normal-5477.png,Normal/masks/Normal-5477.png,True,G14581,val,e2d3a224e930ec2c7a93de0e126884d4,d17b46861e194d4f
Normal-5478.png,Normal,Normal/images/Normal-5478.png,Normal/masks/Normal-5478.png,True,G14582,val,7df62340bf70701a5f30c1c752172ae7,987b071e1a4d7b38
Normal-5479.png,Normal,Normal/images/Normal-5479.png,Normal/masks/Normal-5479.png,True,G14583,train,54134b792feeba135b54ccbf68a3d9c0,c83d56c21b1c6f93
Normal-548.png,Normal,Normal/images/Normal-548.png,Normal/masks/Normal-548.png,True,G14584,train,85c7887d8940609391c7dba0dc799374,90bad65a94c8bdb4
Normal-5480.png,Normal,Normal/images/Normal-5480.png,Normal/masks/Normal-5480.png,True,G14585,test,50912fcecbc5a625582224bc9be44116,d87dc45694c61e3c
Normal-5481.png,Normal,Normal/images/Normal-5481.png,Normal/masks/Normal-5481.png,True,G14586,train,05105931cebb31e742cfa61431b95333,dd4c53030d2e3cbe
Normal-5482.png,Normal,Normal/images/Normal-5482.png,Normal/masks/Normal-5482.png,True,G14587,train,0c96ed15a3643dd1984d05a202c45da2,c42dc75d0c4caf9a
Normal-5483.png,Normal,Normal/images/Normal-5483.png,Normal/masks/Normal-5483.png,True,G14588,val,af378db17eae47539bd37255fc9b4134,d93bd21f920a1be2
Normal-5484.png,Normal,Normal/images/Normal-5484.png,Normal/masks/Normal-5484.png,True,G14589,train,30fb3645c77d36374c5f295d8106b80f,d925061b457ae1ee
Normal-5485.png,Normal,Normal/images/Normal-5485.png,Normal/masks/Normal-5485.png,True,G14590,train,029e53ab21d6452d7552557da85f1d74,d03d53574b4c4b4b
Normal-5486.png,Normal,Normal/images/Normal-5486.png,Normal/masks/Normal-5486.png,True,G14591,train,2be71126abfa57dc9f5a693e8f377ef5,d52d020b3a6f3e36
Normal-5487.png,Normal,Normal/images/Normal-5487.png,Normal/masks/Normal-5487.png,True,G14592,test,d7721fd613080bde052e29476a860f74,dd6d821e0d4f1b45
Normal-5488.png,Normal,Normal/images/Normal-5488.png,Normal/masks/Normal-5488.png,True,G14593,train,9b1719984921fc23b28327b300633aff,d97f871a054e3a49
Normal-5489.png,Normal,Normal/images/Normal-5489.png,Normal/masks/Normal-5489.png,True,G14594,train,d06355a630d0acc51cfad1510d5f3292,9979871e081f1a5f
Normal-549.png,Normal,Normal/images/Normal-549.png,Normal/masks/Normal-549.png,True,G14595,train,accd60b0f3cd9c5bc355bcb3bce51310,813ec71f4d487c4d
Normal-5490.png,Normal,Normal/images/Normal-5490.png,Normal/masks/Normal-5490.png,True,G14596,train,03b7722fbe2eed0e20bd3e6a7a4ac1a8,817e96101e5f1f1e
Normal-5491.png,Normal,Normal/images/Normal-5491.png,Normal/masks/Normal-5491.png,True,G14597,train,68ed259902194dd5c8e4f2a842878495,dc2e438f0ca4bf94
Normal-5492.png,Normal,Normal/images/Normal-5492.png,Normal/masks/Normal-5492.png,True,G14598,train,c458b7e5034faff714b8f957630e6fb6,d929961e5d487b19
Normal-5493.png,Normal,Normal/images/Normal-5493.png,Normal/masks/Normal-5493.png,True,G14599,val,a129025735bfd3d3f51b64513d7a1fc6,887f061e1a4f5f19
Normal-5494.png,Normal,Normal/images/Normal-5494.png,Normal/masks/Normal-5494.png,True,G14600,val,707c1e489fe07a5e31cb1d5c0c51f20d,c93ac60a1bc66bb9
Normal-5495.png,Normal,Normal/images/Normal-5495.png,Normal/masks/Normal-5495.png,True,G14601,test,50dbff5e6ca1ea26c4c7e521df15a890,897d861a095f3b47
Normal-5496.png,Normal,Normal/images/Normal-5496.png,Normal/masks/Normal-5496.png,True,G14602,test,17cd94fa0cc54a2fab473b5c1b79a592,d93bd6461a454f19
Normal-5497.png,Normal,Normal/images/Normal-5497.png,Normal/masks/Normal-5497.png,True,G14603,train,2b0a8b6d7130bf85265542423908016e,817b46561a4fe939
Normal-5498.png,Normal,Normal/images/Normal-5498.png,Normal/masks/Normal-5498.png,True,G14604,train,9411570b0257cdf38e29df983b10182a,dd39d20f18481f9d
Normal-5499.png,Normal,Normal/images/Normal-5499.png,Normal/masks/Normal-5499.png,True,G14605,train,b9198417e6be5d31af42492ac9079454,d42e82d35b424f1f
Normal-55.png,Normal,Normal/images/Normal-55.png,Normal/masks/Normal-55.png,True,G14606,train,75c0443f327b7f239d52a7b38573a212,884fc75b5d6c282d
Normal-550.png,Normal,Normal/images/Normal-550.png,Normal/masks/Normal-550.png,True,G14607,train,5651e8e95d6991408b6d577ce6e351a2,812b865ad85979d7
Normal-5500.png,Normal,Normal/images/Normal-5500.png,Normal/masks/Normal-5500.png,True,G14608,train,413451dded1bf74884ff8199a8adb574,883d461a5d5b3dca
Normal-5501.png,Normal,Normal/images/Normal-5501.png,Normal/masks/Normal-5501.png,True,G14609,train,19ea52c53efaa01b907f65aafbaa1b09,997e071a471f1955
Normal-5502.png,Normal,Normal/images/Normal-5502.png,Normal/masks/Normal-5502.png,True,G14610,train,f5677a6633e59fab1cd3214dce2567dd,9939c61e15c71fb0
Normal-5503.png,Normal,Normal/images/Normal-5503.png,Normal/masks/Normal-5503.png,True,G14611,train,cec8d56c5f5af51bafe8e0dca22c93ff,d029d35f5c40791f
Normal-5504.png,Normal,Normal/images/Normal-5504.png,Normal/masks/Normal-5504.png,True,G14612,train,82f7573026b3820faa66eb93bced4863,917907861a497b7d
Normal-5505.png,Normal,Normal/images/Normal-5505.png,Normal/masks/Normal-5505.png,True,G14613,train,94fb4e36f6a315c7bd87caba8ad25cee,d42c470b0e4f6f2e
Normal-5506.png,Normal,Normal/images/Normal-5506.png,Normal/masks/Normal-5506.png,True,G14614,train,7f0ddd13e5baf2f68b5c63307a1292a3,d93d86021b475f1b
Normal-5507.png,Normal,Normal/images/Normal-5507.png,Normal/masks/Normal-5507.png,True,G14615,test,e78309c9555d29749d5f0ca4268253c3,cd34471a4c4f5a4f
Normal-5508.png,Normal,Normal/images/Normal-5508.png,Normal/masks/Normal-5508.png,True,G14616,train,b0968b8231fba37c2217c94ae240d08c,b97d861e1517184f
Normal-5509.png,Normal,Normal/images/Normal-5509.png,Normal/masks/Normal-5509.png,True,G14617,train,94e8c99d10f49ccccee92caaac69cbfe,997f0616185b1b5e
Normal-551.png,Normal,Normal/images/Normal-551.png,Normal/masks/Normal-551.png,True,G14618,val,52a2e9cc9a48dc028c4bb3d6fc35f03e,8454a71bfb2d58a5
Normal-5510.png,Normal,Normal/images/Normal-5510.png,Normal/masks/Normal-5510.png,True,G14619,test,f77206acdbed75ffb456b5e3ca899c4d,f879069e1ec65a98
Normal-5511.png,Normal,Normal/images/Normal-5511.png,Normal/masks/Normal-5511.png,True,G14620,val,2fe12c030838ca93f45d334eac0ddf50,997d061a155f194f
Normal-5512.png,Normal,Normal/images/Normal-5512.png,Normal/masks/Normal-5512.png,True,G14621,val,21971cc9fa74b39129b4d455f70659bd,d42e42d70f4cee8a
Normal-5513.png,Normal,Normal/images/Normal-5513.png,Normal/masks/Normal-5513.png,True,G14622,train,8ebe8dbb1881ceb0b73f128dd7e92bfc,9979071e584f5c78
Normal-5514.png,Normal,Normal/images/Normal-5514.png,Normal/masks/Normal-5514.png,True,G14623,train,f29eb93f400228ef448c60600626385f,d52dc25f524b1c5a
Normal-5515.png,Normal,Normal/images/Normal-5515.png,Normal/masks/Normal-5515.png,True,G14624,test,6fe5ed8e016fc9161e9fd4f01a61d296,d969c61f11455b39
Normal-5516.png,Normal,Normal/images/Normal-5516.png,Normal/masks/Normal-5516.png,True,G14625,val,b86534cbb58769d0b7d6e972f2333ded,987f45561f47461a
Normal-5517.png,Normal,Normal/images/Normal-5517.png,Normal/masks/Normal-5517.png,True,G14626,train,4157e48fcb26cf3068bd452209ac6622,d0ffd25fd2435840
Normal-5518.png,Normal,Normal/images/Normal-5518.png,Normal/masks/Normal-5518.png,True,G14627,val,bb9a952e865b1b84b2a59ada754e052f,887f071a1d0f1b5d
Normal-5519.png,Normal,Normal/images/Normal-5519.png,Normal/masks/Normal-5519.png,True,G14628,train,3b05696dcc782f691bdb44fbafa59ca9,913c4e191bee6cb1
Normal-552.png,Normal,Normal/images/Normal-552.png,Normal/masks/Normal-552.png,True,G14629,train,515c6f41164813dacb609d806c0682c7,952f871e0b1d6c4d
Normal-5520.png,Normal,Normal/images/Normal-5520.png,Normal/masks/Normal-5520.png,True,G14630,train,e1cbac1ff0807415bdbd5e9056858132,817a161e185fe97a
Normal-5521.png,Normal,Normal/images/Normal-5521.png,Normal/masks/Normal-5521.png,True,G14631,train,ce55931470c1e0be74a59458a2f88c21,d92987861f075f16
Normal-5522.png,Normal,Normal/images/Normal-5522.png,Normal/masks/Normal-5522.png,True,G14632,test,23e72f228ee268fc966ada06e4efc403,b805061e5d1e5f5e
Normal-5523.png,Normal,Normal/images/Normal-5523.png,Normal/masks/Normal-5523.png,True,G14633,train,6b0345234fd4fb9a5de83da8d8077ea0,cdf8d25dc20b18cb
Normal-5524.png,Normal,Normal/images/Normal-5524.png,Normal/masks/Normal-5524.png,True,G14634,train,e4848aa16b6d6d61d6bb6a845391a00c,dc2ec25f4f4c0d1c
Normal-5525.png,Normal,Normal/images/Normal-5525.png,Normal/masks/Normal-5525.png,True,G14635,test,8202472e280b5019e22eb1e0a372315b,d42cc2431f1e4f2f
Normal-5526.png,Normal,Normal/images/Normal-5526.png,Normal/masks/Normal-5526.png,True,G14636,train,bd53bcff5f6fdc80aef63bf18f202500,d939c61e185b5b1c
Normal-5527.png,Normal,Normal/images/Normal-5527.png,Normal/masks/Normal-5527.png,True,G14637,train,66b86ac86f1edddd2218f6fd46b914d9,d42dd34b0f4a0d1f
Normal-5528.png,Normal,Normal/images/Normal-5528.png,Normal/masks/Normal-5528.png,True,G14638,train,ca5ab09ee213c3243fe98486595fc9da,817b841e1a5dd55b
Normal-5529.png,Normal,Normal/images/Normal-5529.png,Normal/masks/Normal-5529.png,True,G14639,train,dc913e5bc51bfc9b39e01aff7ed653f6,993b861e5d51fa41
Normal-553.png,Normal,Normal/images/Normal-553.png,Normal/masks/Normal-553.png,True,G14640,train,d7d6ebf51665ed1ca202c03cba8a6d72,d46fc35d450268e7
Normal-5530.png,Normal,Normal/images/Normal-5530.png,Normal/masks/Normal-5530.png,True,G14641,train,07f416204916fdcc08859a618d738254,987bc5561a494d3b
Normal-5531.png,Normal,Normal/images/Normal-5531.png,Normal/masks/Normal-5531.png,True,G14642,val,dd85fe2ebeaf47fa1421b068bbfc98cb,d53d82071b4e7a5a
Normal-5532.png,Normal,Normal/images/Normal-5532.png,Normal/masks/Normal-5532.png,True,G14643,val,9197e8c22b71183349cf6efb6f245b1c,9d2c86185f4f78c3
Normal-5533.png,Normal,Normal/images/Normal-5533.png,Normal/masks/Normal-5533.png,True,G14644,test,8ec7c15d92c9c9d2317819b2eb2434cc,dd7cc35b490e2e28
Normal-5534.png,Normal,Normal/images/Normal-5534.png,Normal/masks/Normal-5534.png,True,G14645,train,66b316464f3c0dea0136b9d29f8887c6,817f061e1c5b1b1f
Normal-5535.png,Normal,Normal/images/Normal-5535.png,Normal/masks/Normal-5535.png,True,G14646,train,332fdaf52a057df499a43a6c237edba9,dd2ed4430b1b4d4e
Normal-5536.png,Normal,Normal/images/Normal-5536.png,Normal/masks/Normal-5536.png,True,G14647,train,27f773c8165e599e21458d05e603a862,d16b46061a975f1e
Normal-5537.png,Normal,Normal/images/Normal-5537.png,Normal/masks/Normal-5537.png,True,G14648,train,d32832b3cfd1ddaf03783696a80d15cf,836a5c1d1d9d7893
Normal-5538.png,Normal,Normal/images/Normal-5538.png,Normal/masks/Normal-5538.png,True,G14649,train,410a88037696e6c082d63320440653a2,917946c75919fe0a
Normal-5539.png,Normal,Normal/images/Normal-5539.png,Normal/masks/Normal-5539.png,True,G14650,train,2f13b99a950ffb023887eef4bc93d901,e83fc20f0a5f1a2d
Normal-554.png,Normal,Normal/images/Normal-554.png,Normal/masks/Normal-554.png,True,G14651,train,f5bab22bfc4044e5c5fc1cb62c438e75,d47e871fcb0560c6
Normal-5540.png,Normal,Normal/images/Normal-5540.png,Normal/masks/Normal-5540.png,True,G14652,train,e6ccdeb1e99b742690efd8c7c6be7db5,d86dc61a494f1e4d
Normal-5541.png,Normal,Normal/images/Normal-5541.png,Normal/masks/Normal-5541.png,True,G14653,train,2fd231187a8fd11e0e7186334d9a7324,c13d06c3135f4b4f
Normal-5542.png,Normal,Normal/images/Normal-5542.png,Normal/masks/Normal-5542.png,True,G14654,train,52cfe7cced9778795ff3092f77a49cd9,d93b55571a470a1d
Normal-5543.png,Normal,Normal/images/Normal-5543.png,Normal/masks/Normal-5543.png,True,G14655,train,3a38d0179ccf946ed509a4dac7a0a3db,dd2ad20b1f452b1d
Normal-5544.png,Normal,Normal/images/Normal-5544.png,Normal/masks/Normal-5544.png,True,G14656,train,5fc9c152c62c5a8530ce9bdc3c4d7efb,d979c65f044a1b55
Normal-5545.png,Normal,Normal/images/Normal-5545.png,Normal/masks/Normal-5545.png,True,G14657,train,25cbdc4bfc3e4f15b3b6c2f5159279ab,dd2c520b5f0d6c2d
Normal-5546.png,Normal,Normal/images/Normal-5546.png,Normal/masks/Normal-5546.png,True,G14658,train,d19240afef2eb62387f5a94ecc8adb1b,9169861e1b8f6aba
Normal-5547.png,Normal,Normal/images/Normal-5547.png,Normal/masks/Normal-5547.png,True,G14659,val,d82c505394b630379da49df158f02db3,c2262f31323b3d6e
Normal-5548.png,Normal,Normal/images/Normal-5548.png,Normal/masks/Normal-5548.png,True,G14660,val,45e7fa2b0e9bf4a7d2cfb448edb3de42,dc3dc71b504a2f29
Normal-5549.png,Normal,Normal/images/Normal-5549.png,Normal/masks/Normal-5549.png,True,G14661,test,7893bdf5e87504a841af9a99dfdf1d8b,d42e535b4b4a6b1a
Normal-555.png,Normal,Normal/images/Normal-555.png,Normal/masks/Normal-555.png,True,G14662,val,181ae963e91f7e59b18d1267851c4032,906dc718cd6d18cf
Normal-5550.png,Normal,Normal/images/Normal-5550.png,Normal/masks/Normal-5550.png,True,G14663,test,041cef2549ac05d3f2d9682e935ac792,d428d2535f524f0f
Normal-5551.png,Normal,Normal/images/Normal-5551.png,Normal/masks/Normal-5551.png,True,G14664,train,1b6d8277c332b91fb6f70b9184f943a8,d97a461b192c2fb8
Normal-5552.png,Normal,Normal/images/Normal-5552.png,Normal/masks/Normal-5552.png,True,G14665,train,5e44437b0de7e6f2c7f3016f130cede8,f969861e1c1d1b4d
Normal-5553.png,Normal,Normal/images/Normal-5553.png,Normal/masks/Normal-5553.png,True,G14666,train,c934209f4e2bdf156b2c9118b350b497,fd3c574b0f4e0a0a
Normal-5554.png,Normal,Normal/images/Normal-5554.png,Normal/masks/Normal-5554.png,True,G14667,test,2c4a7e71ecf63b8076088da5a531c4ea,99688619174f3b9b
Normal-5555.png,Normal,Normal/images/Normal-5555.png,Normal/masks/Normal-5555.png,True,G14668,test,7207caee6a47579a13697cbc3aaf54c7,c43d4b4f4b0b2c6d
Normal-5556.png,Normal,Normal/images/Normal-5556.png,Normal/masks/Normal-5556.png,True,G14669,test,b1990fe9a2bebb71372e32e14a05ed56,d1f9e61e04c33ac3
Normal-5557.png,Normal,Normal/images/Normal-5557.png,Normal/masks/Normal-5557.png,True,G14670,train,4cb8266f188285834e5cfdc2156b1b5e,d97dc7561a455a18
Normal-5558.png,Normal,Normal/images/Normal-5558.png,Normal/masks/Normal-5558.png,True,G14671,train,20583373a1712cfa396dcec0e4c92b9c,c97ad71fd24a1a0a
Normal-5559.png,Normal,Normal/images/Normal-5559.png,Normal/masks/Normal-5559.png,True,G14672,train,d22eaa80fa0ca8c851f1a17f4556c674,c13d460f196e69b9
Normal-556.png,Normal,Normal/images/Normal-556.png,Normal/masks/Normal-556.png,True,G14673,train,8f2cc1ce28abb21bb17142b4004e8db7,d12dc61be96938c9
Normal-5560.png,Normal,Normal/images/Normal-5560.png,Normal/masks/Normal-5560.png,True,G14674,train,d9020afb3d4416eefe3a1d62b5a28f94,8939861a1d597bbc
Normal-5561.png,Normal,Normal/images/Normal-5561.png,Normal/masks/Normal-5561.png,True,G14675,train,fc1e7293feafec21d2ee5aed92503b2a,993fc61e1c4a394d
Normal-5562.png,Normal,Normal/images/Normal-5562.png,Normal/masks/Normal-5562.png,True,G14676,train,09934181a35cf25591a4dc4ac186faf1,d968561f56471bc2
Normal-5563.png,Normal,Normal/images/Normal-5563.png,Normal/masks/Normal-5563.png,True,G14677,train,a3653b447c9ce307b5d52bfcff7a6eaf,dd2cc20bd14f1d47
Normal-5564.png,Normal,Normal/images/Normal-5564.png,Normal/masks/Normal-5564.png,True,G14678,train,68cefba6462d4def93383f0a8e670aca,996806161b4d5fbb
Normal-5565.png,Normal,Normal/images/Normal-5565.png,Normal/masks/Normal-5565.png,True,G14679,train,a9a2c9ed8b49e4c464ffbb45ca0b2608,d42c42c35f1d4d6e
Normal-5566.png,Normal,Normal/images/Normal-5566.png,Normal/masks/Normal-5566.png,True,G14680,train,7b4f9aa2c2652f295e2a4250a99b95d9,817a2f07961f7949
Normal-5567.png,Normal,Normal/images/Normal-5567.png,Normal/masks/Normal-5567.png,True,G14681,val,4cf029936f464063ba934bf2a0eeaca4,dc2dd24b1f4c091f
Normal-5568.png,Normal,Normal/images/Normal-5568.png,Normal/masks/Normal-5568.png,True,G14682,val,2871f69f22701b540c283002259e040c,d86ac31f0c2a9fa5
Normal-5569.png,Normal,Normal/images/Normal-5569.png,Normal/masks/Normal-5569.png,True,G14683,val,3656cc5fe7c23db59934fed5bd333aca,897b861e197b19b8
Normal-557.png,Normal,Normal/images/Normal-557.png,Normal/masks/Normal-557.png,True,G14684,test,ecdef0c4a5fadb98f3a5ebc8a967b412,913ada1a181e3ede
Normal-5570.png,Normal,Normal/images/Normal-5570.png,Normal/masks/Normal-5570.png,True,G14685,train,3fc4ffb1f444f5f3c92e353f024edcb4,887f85171e1c1f1b
Normal-5571.png,Normal,Normal/images/Normal-5571.png,Normal/masks/Normal-5571.png,True,G14686,train,f08ac23d60e3c22a15a61613741cba87,dc2d42430f4aad3f
Normal-5572.png,Normal,Normal/images/Normal-5572.png,Normal/masks/Normal-5572.png,True,G14687,train,a1b20e2ff5e6440130e301e7a0a231da,d938c64a1a4f3a79
Normal-5573.png,Normal,Normal/images/Normal-5573.png,Normal/masks/Normal-5573.png,True,G14688,train,6ea634c336924aaba18a9d6c504c4b3f,997f170a1e057a99
Normal-5574.png,Normal,Normal/images/Normal-5574.png,Normal/masks/Normal-5574.png,True,G14689,train,de3138826764d27c82960bffecb9e2f0,d47d4257594a4f2c
Normal-5575.png,Normal,Normal/images/Normal-5575.png,Normal/masks/Normal-5575.png,True,G14690,train,4fb7be8db6303e7239ac940a105c0021,cd3cc20b1d0d1fad
Normal-5576.png,Normal,Normal/images/Normal-5576.png,Normal/masks/Normal-5576.png,True,G14691,train,94a88897d4a676be0a2735ac4eeeb722,c42cc3535d4f2f38
Normal-5577.png,Normal,Normal/images/Normal-5577.png,Normal/masks/Normal-5577.png,True,G14692,train,cf0c985ce9ed5dcb5f18ca4cce5252dc,9d3c530bc51abde0
Normal-5578.png,Normal,Normal/images/Normal-5578.png,Normal/masks/Normal-5578.png,True,G14693,train,123a0b57bd9167d0054e1172f4a89ae7,dd29d2831f554a1d
Normal-5579.png,Normal,Normal/images/Normal-5579.png,Normal/masks/Normal-5579.png,True,G14694,train,3340b148efbcef7114daa62e2cf2169b,816807165a5f7a7b
Normal-558.png,Normal,Normal/images/Normal-558.png,Normal/masks/Normal-558.png,True,G14695,train,8aaf0e551ad682349831ac3a6e3c6b0c,d03cc73bb96d3c20
Normal-5580.png,Normal,Normal/images/Normal-5580.png,Normal/masks/Normal-5580.png,True,G14696,train,5dca6c2ff1047bfd8e4e58b9dba7507f,9908075e5f7879e8
Normal-5581.png,Normal,Normal/images/Normal-5581.png,Normal/masks/Normal-5581.png,True,G14697,train,24afa3eb6fcdf6127c029ff593def9e6,8979e5461e84ce9e
Normal-5582.png,Normal,Normal/images/Normal-5582.png,Normal/masks/Normal-5582.png,True,G14698,train,d9887de603646a846f21ace6496dfb08,cc3cd24b024f3f6c
Normal-5583.png,Normal,Normal/images/Normal-5583.png,Normal/masks/Normal-5583.png,True,G14699,val,1691b922bc92e9ecc0864b466ad57b6c,9d2c86c61a59ef16
Normal-5584.png,Normal,Normal/images/Normal-5584.png,Normal/masks/Normal-5584.png,True,G14700,val,d2e07839f85a73513bddc407430a550d,8938c71b13475e5b
Normal-5585.png,Normal,Normal/images/Normal-5585.png,Normal/masks/Normal-5585.png,True,G14701,test,78bc6d2b3060a10aae038d8950f1cd3b,883d461e1e4f1e6b
Normal-5586.png,Normal,Normal/images/Normal-5586.png,Normal/masks/Normal-5586.png,True,G14702,val,633233bdae8df7c2ff76076a8a91e0e1,d17f425753424b1d
Normal-5587.png,Normal,Normal/images/Normal-5587.png,Normal/masks/Normal-5587.png,True,G14703,test,d7de30e6f4f8e8a6e5646838752ae45b,9925421bed7ab8e2
Normal-5588.png,Normal,Normal/images/Normal-5588.png,Normal/masks/Normal-5588.png,True,G14704,train,ddf865087a9d1f330f3204b72357bc0a,9d3d80171b450f5f
Normal-5589.png,Normal,Normal/images/Normal-5589.png,Normal/masks/Normal-5589.png,True,G14705,test,1f2e66552a100526d39aee6174120d85,9d0d871b0d5e1d47
Normal-559.png,Normal,Normal/images/Normal-559.png,Normal/masks/Normal-559.png,True,G14706,train,35226eba10cc9ab648e8d9cd4a26f7a0,903bd21abd589ad6
Normal-5590.png,Normal,Normal/images/Normal-5590.png,Normal/masks/Normal-5590.png,True,G14707,train,606cb56529690340633fca197441f2a8,d96907961dcb2b91
Normal-5591.png,Normal,Normal/images/Normal-5591.png,Normal/masks/Normal-5591.png,True,G14708,train,b9c8fdc1496b870519112169bbe27349,fd2c531b174b1d44
Normal-5592.png,Normal,Normal/images/Normal-5592.png,Normal/masks/Normal-5592.png,True,G14709,train,63375f005cdf0ca313b39b3cc6b43ed2,992dc71a194d1e79
Normal-5593.png,Normal,Normal/images/Normal-5593.png,Normal/masks/Normal-5593.png,True,G14710,train,1ffa5f88b167c092c19dddfedfa3797a,817816861bd9fa79
Normal-5594.png,Normal,Normal/images/Normal-5594.png,Normal/masks/Normal-5594.png,True,G14711,train,a70f3349c0e643b3a8bb4343ba60a7dc,91b9c614974f1557
Normal-5595.png,Normal,Normal/images/Normal-5595.png,Normal/masks/Normal-5595.png,True,G14712,train,7ff41556683c2a3a7554ca05fc0a600e,817b1c185b5fe859
Normal-5596.png,Normal,Normal/images/Normal-5596.png,Normal/masks/Normal-5596.png,True,G14713,train,8d9b21c2f78d5d2195c22248c7f43e70,d56e020b4a39ef4d
Normal-5597.png,Normal,Normal/images/Normal-5597.png,Normal/masks/Normal-5597.png,True,G14714,train,8894b04d554c4af4321d62284fcef82a,893cc35f564e1c4b
Normal-5598.png,Normal,Normal/images/Normal-5598.png,Normal/masks/Normal-5598.png,True,G14715,train,a729060f1606e7ad28e39880dabffb3b,89688e1a1b5e7a7a
Normal-5599.png,Normal,Normal/images/Normal-5599.png,Normal/masks/Normal-5599.png,True,G14716,train,75f9f22756ad60bc2abdc2d19f0f6348,9961861a5b5e7939
Normal-56.png,Normal,Normal/images/Normal-56.png,Normal/masks/Normal-56.png,True,G14717,val,62637df4a1cec5c48f03898cabd32076,9d7e421b474b504f
Normal-560.png,Normal,Normal/images/Normal-560.png,Normal/masks/Normal-560.png,True,G14718,train,876f3131e2489138d11c167aa05b8457,917fce1a184b195d
Normal-5600.png,Normal,Normal/images/Normal-5600.png,Normal/masks/Normal-5600.png,True,G14719,train,b2010731dd2983db8530cfc8611139e0,9968861b974e39ce
Normal-5601.png,Normal,Normal/images/Normal-5601.png,Normal/masks/Normal-5601.png,True,G14720,val,0c9ec0df64bf813fe3a1cee40d2432bb,d93d861a55493be1
Normal-5602.png,Normal,Normal/images/Normal-5602.png,Normal/masks/Normal-5602.png,True,G14721,val,dd69d24a09603f5fcac3a8e117b6a5ce,99280e1a1b5d7f78
Normal-5603.png,Normal,Normal/images/Normal-5603.png,Normal/masks/Normal-5603.png,True,G14722,train,6e0675588be83d2c2f8015effc047bd4,99394e161d465e79
Normal-5604.png,Normal,Normal/images/Normal-5604.png,Normal/masks/Normal-5604.png,True,G14723,test,d7b0880aea60c0d0f74f3f76ca4cf751,dd2cc20b5c6a2df4
Normal-5605.png,Normal,Normal/images/Normal-5605.png,Normal/masks/Normal-5605.png,True,G14724,train,ba9e87863f7d616ef100efea9a6256ff,c93c06431cecedb9
Normal-5606.png,Normal,Normal/images/Normal-5606.png,Normal/masks/Normal-5606.png,True,G14725,test,80648aa297956f8b3b919f3a55bcb1d4,fd4d43130b693cb8
Normal-5607.png,Normal,Normal/images/Normal-5607.png,Normal/masks/Normal-5607.png,True,G14726,test,9a4909706cb0c93663d69b0548707909,c03d4b420cadafbd
Normal-5608.png,Normal,Normal/images/Normal-5608.png,Normal/masks/Normal-5608.png,True,G14727,train,e042ef9fc73933786742a4ab2d1981cf,9178d21b0eaff8a4
Normal-5609.png,Normal,Normal/images/Normal-5609.png,Normal/masks/Normal-5609.png,True,G14728,val,20667b8f178435d0acafd2a72c4e228e,83681c3b6f7e254c
Normal-561.png,Normal,Normal/images/Normal-561.png,Normal/masks/Normal-561.png,True,G14729,train,e2bc264f01101958d5af7d5cdab4c19b,d03dc23b98693df0
Normal-5610.png,Normal,Normal/images/Normal-5610.png,Normal/masks/Normal-5610.png,True,G14730,test,6dc6ac1967187519ee7f16011486cdcb,917906865a1fce1f
Normal-5611.png,Normal,Normal/images/Normal-5611.png,Normal/masks/Normal-5611.png,True,G14731,val,a8bbf02f1a781dfb4d85c3d4e6c30c4f,992dc65e0e4d5a2d
Normal-5612.png,Normal,Normal/images/Normal-5612.png,Normal/masks/Normal-5612.png,True,G14732,train,e8d622befaf741365ecc0de08bfd73fd,9979c5161a815f79
Normal-5613.png,Normal,Normal/images/Normal-5613.png,Normal/masks/Normal-5613.png,True,G14733,train,3ff0e22244a6c5513f18fac8f9dcbfc9,897816181b5f7979
Normal-5614.png,Normal,Normal/images/Normal-5614.png,Normal/masks/Normal-5614.png,True,G14734,train,e300b0f6c8062ccebd108cef95e382ec,dc2f535f4c0b0c47
Normal-5615.png,Normal,Normal/images/Normal-5615.png,Normal/masks/Normal-5615.png,True,G14735,train,340cb679810e1432125138b4b824ce25,dd38c21b1d4a1fca
Normal-5616.png,Normal,Normal/images/Normal-5616.png,Normal/masks/Normal-5616.png,True,G14736,train,7927894a80a112853f57c623b95c679b,887f971e041a555f
Normal-5617.png,Normal,Normal/images/Normal-5617.png,Normal/masks/Normal-5617.png,True,G14737,train,4c44ce57564cd21328f1c38b49e9afac,b9798616194e5f1a
Normal-5618.png,Normal,Normal/images/Normal-5618.png,Normal/masks/Normal-5618.png,True,G14738,train,b33ce2808c7da05470734055f722945c,fd0c825b1c4c1dfc
Normal-5619.png,Normal,Normal/images/Normal-5619.png,Normal/masks/Normal-5619.png,True,G14739,train,00b85b921f79c4224023abfcbac561bb,cc2f434d4d2d3ce4
Normal-562.png,Normal,Normal/images/Normal-562.png,Normal/masks/Normal-562.png,True,G14740,train,505bb653a8bd4abf39bb12961973e20b,907ce73abcc93988
Normal-5620.png,Normal,Normal/images/Normal-5620.png,Normal/masks/Normal-5620.png,True,G14741,train,fdf4ffbc55aac420a8070ee6549cff0e,9919061a59797dce
Normal-5621.png,Normal,Normal/images/Normal-5621.png,Normal/masks/Normal-5621.png,True,G14742,val,fdac2ad393c0767793c050613d01c51e,dd2fd20f083d0c9d
Normal-5622.png,Normal,Normal/images/Normal-5622.png,Normal/masks/Normal-5622.png,True,G14743,test,912215126df72243a1cb9d20c4906a12,dd28d65f5c4a1d0a
Normal-5623.png,Normal,Normal/images/Normal-5623.png,Normal/masks/Normal-5623.png,True,G14744,test,fbf76e33fc658b3d73742eeb52245190,d12d061c3b6ce9b9
Normal-5624.png,Normal,Normal/images/Normal-5624.png,Normal/masks/Normal-5624.png,True,G14745,train,61a85988da3a9ea1418c8c866e3ca34f,d02947d61fc5669c
Normal-5625.png,Normal,Normal/images/Normal-5625.png,Normal/masks/Normal-5625.png,True,G14746,train,1cc5d3ca705736009656a6932ec41c6f,81781c9e1a5b597b
Normal-5626.png,Normal,Normal/images/Normal-5626.png,Normal/masks/Normal-5626.png,True,G14747,train,0191d308f68819a3a6d93e6b5bf9e7fb,9968861a1bcf3e59
Normal-5627.png,Normal,Normal/images/Normal-5627.png,Normal/masks/Normal-5627.png,True,G14748,train,ba3f7bddc26b1ce76e2451fe5471f5b7,897d061a1d4b3be9
Normal-5628.png,Normal,Normal/images/Normal-5628.png,Normal/masks/Normal-5628.png,True,G14749,train,02582b5fd66ab7725038f3a4b4496a83,c969861e1e5f5a58
Normal-5629.png,Normal,Normal/images/Normal-5629.png,Normal/masks/Normal-5629.png,True,G14750,train,6232b044cd93b376bf9b2bab7b2ea038,906c02571b5def1b
Normal-563.png,Normal,Normal/images/Normal-563.png,Normal/masks/Normal-563.png,True,G14751,train,a53cc84f2695365dbe0ab27267c76a53,90ed871abb3f3824
Normal-5630.png,Normal,Normal/images/Normal-5630.png,Normal/masks/Normal-5630.png,True,G14752,train,4eb935ec4970f07953e6c029461c66e8,d43c03c91fece4d2
Normal-5631.png,Normal,Normal/images/Normal-5631.png,Normal/masks/Normal-5631.png,True,G14753,train,65fb32be72009b83eca44ef7135d6855,dd5f575b1d4c0412
Normal-5632.png,Normal,Normal/images/Normal-5632.png,Normal/masks/Normal-5632.png,True,G14754,train,c7aecca594f2913a3b280580c4638124,dd29c31b1c4d5f18
Normal-5633.png,Normal,Normal/images/Normal-5633.png,Normal/masks/Normal-5633.png,True,G14755,train,b6e4afe689670ebe7f9f2d193c544a6d,817987161a5d6f59
Normal-5634.png,Normal,Normal/images/Normal-5634.png,Normal/masks/Normal-5634.png,True,G14756,train,3c58c39c0bea7c6616a1c1c0aa25ee65,d93d46461b1f4a0f
Normal-5635.png,Normal,Normal/images/Normal-5635.png,Normal/masks/Normal-5635.png,True,G14757,val,f75af5b9e2eb5dc2baf4075250843461,c0c43df3c32f4bc8
Normal-5636.png,Normal,Normal/images/Normal-5636.png,Normal/masks/Normal-5636.png,True,G14758,train,83582e3b212e57f3477d1f145f81b626,d5280f4b4d6c7c78
Normal-5637.png,Normal,Normal/images/Normal-5637.png,Normal/masks/Normal-5637.png,True,G14759,val,b22110fff8c09a0e3ca0ea95ee2d8cd1,c97dc61a1d4b191d
Normal-5638.png,Normal,Normal/images/Normal-5638.png,Normal/masks/Normal-5638.png,True,G14760,test,cd98758642a0d9902a8afd09037006c1,cc0e470a596d6fe8
Normal-5639.png,Normal,Normal/images/Normal-5639.png,Normal/masks/Normal-5639.png,True,G14761,train,4284ace68320b2a2d5334fb776f9cd83,883d87161f075f1d
Normal-564.png,Normal,Normal/images/Normal-564.png,Normal/masks/Normal-564.png,True,G14762,test,b59bb784850bcdeee20de5a438c3ec9c,957e87398d2c6986
Normal-5640.png,Normal,Normal/images/Normal-5640.png,Normal/masks/Normal-5640.png,True,G14763,val,ffeaaa65458c5ebfb64ac0c2cc63850b,dd3d86c21a9ec4aa
Normal-5641.png,Normal,Normal/images/Normal-5641.png,Normal/masks/Normal-5641.png,True,G14764,test,842d5376897fe156987f4249a4fe9c86,9538468a1b4f1b5f
Normal-5642.png,Normal,Normal/images/Normal-5642.png,Normal/masks/Normal-5642.png,True,G14765,train,46aee08a6891b520a8202aa26f085def,e979869696c71a19
Normal-5643.png,Normal,Normal/images/Normal-5643.png,Normal/masks/Normal-5643.png,True,G14766,train,23ea947a7955c9480073c3a959c3ac2f,dd68064f5a0e6b78
Normal-5644.png,Normal,Normal/images/Normal-5644.png,Normal/masks/Normal-5644.png,True,G14767,test,a57014d3a2e3ffb438db888123fc8106,957e06531b4c687d
Normal-5645.png,Normal,Normal/images/Normal-5645.png,Normal/masks/Normal-5645.png,True,G14768,train,37db46585347ae311bdb656cf7c86c51,d979458616874757
Normal-5646.png,Normal,Normal/images/Normal-5646.png,Normal/masks/Normal-5646.png,True,G14769,train,74c426c55b494ea574039e2e64facffa,99790619197e7ae1
Normal-5647.png,Normal,Normal/images/Normal-5647.png,Normal/masks/Normal-5647.png,True,G14770,train,2eeeaf25c655d6a53b5f9e8c7e9daa14,d436430b4f1f4b2e
Normal-5648.png,Normal,Normal/images/Normal-5648.png,Normal/masks/Normal-5648.png,True,G14771,train,4e60034940619a82a9e070b21fea7719,8c3a471b456f1a6d
Normal-5649.png,Normal,Normal/images/Normal-5649.png,Normal/masks/Normal-5649.png,True,G14772,train,9d2308d5e95de636ad53e889ea1b8923,c9bdc69a941b904f
Normal-565.png,Normal,Normal/images/Normal-565.png,Normal/masks/Normal-565.png,True,G14773,val,32b1cfbee1a9b4456a3a843a734888d6,d47dc71f9b4d0c04
Normal-5650.png,Normal,Normal/images/Normal-5650.png,Normal/masks/Normal-5650.png,True,G14774,train,005e099b687d8431bfd30eea3fb877cf,dd2cc31b1d4e1d1c
Normal-5651.png,Normal,Normal/images/Normal-5651.png,Normal/masks/Normal-5651.png,True,G14775,train,7c2ec50985d7f6a7f843e825743d3e87,d82dc7d60a4c9f0b
Normal-5652.png,Normal,Normal/images/Normal-5652.png,Normal/masks/Normal-5652.png,True,G14776,train,cb3320f78c84663ab0f04d67adfcbdfd,d42dc3531b46c759
Normal-5653.png,Normal,Normal/images/Normal-5653.png,Normal/masks/Normal-5653.png,True,G14777,train,0e9366dbf1c53a587412a231351dafbb,cc2d025d463bf86d
Normal-5654.png,Normal,Normal/images/Normal-5654.png,Normal/masks/Normal-5654.png,True,G14778,train,cfc4b198e5dd88b0bdcda7d5bf760d74,d42f464f0a2cf783
Normal-5655.png,Normal,Normal/images/Normal-5655.png,Normal/masks/Normal-5655.png,True,G14779,val,df194172eb85979803d07de8594615a7,dd2ad20b5c470f55
Normal-5656.png,Normal,Normal/images/Normal-5656.png,Normal/masks/Normal-5656.png,True,G14780,train,47971837d7218f4831b05f87ee84f668,91695e5e195baa68
Normal-5657.png,Normal,Normal/images/Normal-5657.png,Normal/masks/Normal-5657.png,True,G14781,test,f19e63fc69bf6a0c9e8885c1db4c7fab,997f861a1b452e3a
Normal-5658.png,Normal,Normal/images/Normal-5658.png,Normal/masks/Normal-5658.png,True,G14782,train,88de024c26ffe63dc7da97e8b843e359,d93fc7421a05ea1e
Normal-5659.png,Normal,Normal/images/Normal-5659.png,Normal/masks/Normal-5659.png,True,G14783,train,2908fef3d8088755330ce576aad23958,997947421a1fe93a
Normal-566.png,Normal,Normal/images/Normal-566.png,Normal/masks/Normal-566.png,True,G14784,val,3e370a80f01ffdeec2cdebb57eb74199,907bd65e18511f95
Normal-5660.png,Normal,Normal/images/Normal-5660.png,Normal/masks/Normal-5660.png,True,G14785,train,26bd7662adbd9850ba901b233fe9c481,896dc61b451ebaa5
Normal-5661.png,Normal,Normal/images/Normal-5661.png,Normal/masks/Normal-5661.png,True,G14786,test,cd801be1f2312881cc23ec1643889a2a,d92c861e1c4f78e9
Normal-5662.png,Normal,Normal/images/Normal-5662.png,Normal/masks/Normal-5662.png,True,G14787,val,91d10bac2767efa8568c7f6a6363810d,885947971e9a4bba
Normal-5663.png,Normal,Normal/images/Normal-5663.png,Normal/masks/Normal-5663.png,True,G14788,train,0f998081d7558607bd4d72e60459a228,897905865e1fcc5b
Normal-5664.png,Normal,Normal/images/Normal-5664.png,Normal/masks/Normal-5664.png,True,G14789,train,72d61024d9e8443a83827e78f91300f8,cd3d444b1b0d6d3c
Normal-5665.png,Normal,Normal/images/Normal-5665.png,Normal/masks/Normal-5665.png,True,G14790,train,36b8232f3b2eda1abc01689172d6bae8,cc3d831d563eb84c
Normal-5666.png,Normal,Normal/images/Normal-5666.png,Normal/masks/Normal-5666.png,True,G14791,train,e4d324a5e79ff9427f5b470d51447bd6,d42d021f1b065f7d
Normal-5667.png,Normal,Normal/images/Normal-5667.png,Normal/masks/Normal-5667.png,True,G14792,train,61f7eb8bf3c5b4442037116b02af7ab1,917f441a5f193a0f
Normal-5668.png,Normal,Normal/images/Normal-5668.png,Normal/masks/Normal-5668.png,True,G14793,train,e7bf094b4ffcb478b2f30a2ce569f206,897897185feab8a8
Normal-5669.png,Normal,Normal/images/Normal-5669.png,Normal/masks/Normal-5669.png,True,G14794,train,6e8451e2601f2f6fab3b0d919319313d,997c44c25b45cf1b
Normal-567.png,Normal,Normal/images/Normal-567.png,Normal/masks/Normal-567.png,True,G14795,train,6b95a5df39f7f432bdfd0bc2dd51a9e7,907cc74abdcd38b0
Normal-5670.png,Normal,Normal/images/Normal-5670.png,Normal/masks/Normal-5670.png,True,G14796,train,570c8370092d6c71b760b293a4a43d37,997d44ce1b0d2d1d
Normal-5671.png,Normal,Normal/images/Normal-5671.png,Normal/masks/Normal-5671.png,True,G14797,train,a40bec0189953f2623c3c23c14ad75bd,993907961a9d5f98
Normal-5672.png,Normal,Normal/images/Normal-5672.png,Normal/masks/Normal-5672.png,True,G14798,train,7b1508eadd1c400fdadd0eadeca56af4,dd2cd7421f444b1d
Normal-5673.png,Normal,Normal/images/Normal-5673.png,Normal/masks/Normal-5673.png,True,G14799,train,f671112e8e8d1d964cd842e8b95662df,993d861a0dce3b59
Normal-5674.png,Normal,Normal/images/Normal-5674.png,Normal/masks/Normal-5674.png,True,G14800,train,61c5a969eb2d2637d361811ef6a1fb55,dc0cc34b0c2cecfe
Normal-5675.png,Normal,Normal/images/Normal-5675.png,Normal/masks/Normal-5675.png,True,G14801,val,c9756820fc269a29d04fed3d1f3b1256,dd28d2d754c21fd0
Normal-5676.png,Normal,Normal/images/Normal-5676.png,Normal/masks/Normal-5676.png,True,G14802,test,c487c1a8af90f51db8fa42777313af07,883b0587171f4f1f
Normal-5677.png,Normal,Normal/images/Normal-5677.png,Normal/masks/Normal-5677.png,True,G14803,test,e86f2e0c1e5715fc244734b20ebba7e9,917f475a0f0d1b29
Normal-5678.png,Normal,Normal/images/Normal-5678.png,Normal/masks/Normal-5678.png,True,G14804,test,221bc476c57a4fd25e69e56a81006355,d13946031b1f6d3b
Normal-5679.png,Normal,Normal/images/Normal-5679.png,Normal/masks/Normal-5679.png,True,G14805,val,8bfb7bb126014520ad0b3501706095d9,997dc6121b4f1a59
Normal-568.png,Normal,Normal/images/Normal-568.png,Normal/masks/Normal-568.png,True,G14806,test,ebdb15810e25fd010e7c9de9caf6b4e3,9129c65ad959d993
Normal-5680.png,Normal,Normal/images/Normal-5680.png,Normal/masks/Normal-5680.png,True,G14807,train,5b701807286be43f3375929f3c211e3c,9969d64e1a1b4f29
Normal-5681.png,Normal,Normal/images/Normal-5681.png,Normal/masks/Normal-5681.png,True,G14808,train,69d99115c1db960a01a64751a9248f71,81295e1e587e786b
Normal-5682.png,Normal,Normal/images/Normal-5682.png,Normal/masks/Normal-5682.png,True,G14809,train,62aeca98db4dba355d53e33ef0525481,817987861a1d5d5f
Normal-5683.png,Normal,Normal/images/Normal-5683.png,Normal/masks/Normal-5683.png,True,G14810,train,5a35dae10e22d86dc7dd68435105e4b0,807f061e1a3b3b3b
Normal-5684.png,Normal,Normal/images/Normal-5684.png,Normal/masks/Normal-5684.png,True,G14811,train,2a77d3dfbbd8b8f2cc3f5295d017db0d,997d05861e094f5f
Normal-5685.png,Normal,Normal/images/Normal-5685.png,Normal/masks/Normal-5685.png,True,G14812,train,7176d0981a50bdd5fe861c9f318aa7ec,992b061e5d4f5959
Normal-5686.png,Normal,Normal/images/Normal-5686.png,Normal/masks/Normal-5686.png,True,G14813,train,5316e329c72dcd0c73f457971764861f,dd79d2425f401b1d
Normal-5687.png,Normal,Normal/images/Normal-5687.png,Normal/masks/Normal-5687.png,True,G14814,train,d9d0158ed371534ec0097c05d85bb3c4,b87f861e091b1b1b
Normal-5688.png,Normal,Normal/images/Normal-5688.png,Normal/masks/Normal-5688.png,True,G14815,train,5c7714a988082e9f421ff77b6ad04a0d,dc2c54831b5f4c4f
Normal-5689.png,Normal,Normal/images/Normal-5689.png,Normal/masks/Normal-5689.png,True,G14816,train,487ac1e680373ebfefeafcb73ef2635d,dc2f43570b4a6f18
Normal-569.png,Normal,Normal/images/Normal-569.png,Normal/masks/Normal-569.png,True,G14817,train,5463f450b3eeaa08abf6415ea9ca6bda,903dc71ebe4b3885
Normal-5690.png,Normal,Normal/images/Normal-5690.png,Normal/masks/Normal-5690.png,True,G14818,val,107bbb04179d8ad4fc8b24b34511f346,dd2dc61a0d4a3b4d
Normal-5691.png,Normal,Normal/images/Normal-5691.png,Normal/masks/Normal-5691.png,True,G14819,train,f1dcdff80043871341af5aa5f49dfed0,dc2cc30f0d4e2d3d
Normal-5692.png,Normal,Normal/images/Normal-5692.png,Normal/masks/Normal-5692.png,True,G14820,train,6122e71c353101df58a69368e96b411f,d97a531d4a1d862b
Normal-5693.png,Normal,Normal/images/Normal-5693.png,Normal/masks/Normal-5693.png,True,G14821,train,7ae8e3d7dd49234cd42facc54506c4fe,d03d42471a1fcd2f
Normal-5694.png,Normal,Normal/images/Normal-5694.png,Normal/masks/Normal-5694.png,True,G14822,val,5a0e4ecf64e44732f194d69e1a836aa7,dd2cd24bc24b1d4e
Normal-5695.png,Normal,Normal/images/Normal-5695.png,Normal/masks/Normal-5695.png,True,G14823,val,fb3d81e41eb664b96e5797205806404b,cc3dc24b0c662fbc
Normal-5696.png,Normal,Normal/images/Normal-5696.png,Normal/masks/Normal-5696.png,True,G14824,test,0d2cffbbf85dc422cf8ffdafa48d79f2,81781e9f79d562a1
Normal-5697.png,Normal,Normal/images/Normal-5697.png,Normal/masks/Normal-5697.png,True,G14825,test,b7c24fff3951be77676c180356959694,d92a861f964a1f56
Normal-5698.png,Normal,Normal/images/Normal-5698.png,Normal/masks/Normal-5698.png,True,G14826,train,597d66b2c67baacdf16fe3db47e4bd78,c17ad65a1d5a191d
Normal-5699.png,Normal,Normal/images/Normal-5699.png,Normal/masks/Normal-5699.png,True,G14827,train,8b1ff9d66afa320fb813090a6768b5e9,cd2cc21fc54ab84f
Normal-57.png,Normal,Normal/images/Normal-57.png,Normal/masks/Normal-57.png,True,G14828,train,efa6f8a9a7b269a653926cc5a71e384f,d92917161a5d7978
Normal-570.png,Normal,Normal/images/Normal-570.png,Normal/masks/Normal-570.png,True,G14829,train,6cc92b8a0f32fff1403bd23388591725,917e861ad93d2987
Normal-5700.png,Normal,Normal/images/Normal-5700.png,Normal/masks/Normal-5700.png,True,G14830,test,52106d4d236f9544a71dd454b48ca16a,d93b860a196c7bb8
Normal-5701.png,Normal,Normal/images/Normal-5701.png,Normal/masks/Normal-5701.png,True,G14831,train,d27bad4d34384d6c7a6c8c969004ece2,9979861a155f591d
Normal-5702.png,Normal,Normal/images/Normal-5702.png,Normal/masks/Normal-5702.png,True,G14832,train,86a4c346ed7eb1a3eef294c8b7e5c0f7,9979461e1e4d3aaa
Normal-5703.png,Normal,Normal/images/Normal-5703.png,Normal/masks/Normal-5703.png,True,G14833,train,8ea5f649c65703bf22c896d6d03bfd15,9978c61e194f3a39
Normal-5704.png,Normal,Normal/images/Normal-5704.png,Normal/masks/Normal-5704.png,True,G14834,train,27b9ef9a4872a125e6056ba57b0bfc18,897d861af94df881
Normal-5705.png,Normal,Normal/images/Normal-5705.png,Normal/masks/Normal-5705.png,True,G14835,train,05d8652e0f758722f830302f6261ea40,997887125e4d5b78
Normal-5706.png,Normal,Normal/images/Normal-5706.png,Normal/masks/Normal-5706.png,True,G14836,train,436445720acb1a7eaaba82676595e2c9,997b861e584d3ae1
Normal-5707.png,Normal,Normal/images/Normal-5707.png,Normal/masks/Normal-5707.png,True,G14837,train,3713f8c2037f0db6baec831697874899,d93d44c71a18eb4d
Normal-5708.png,Normal,Normal/images/Normal-5708.png,Normal/masks/Normal-5708.png,True,G14838,train,05343eaa0ecd4a892be2b04314147ae0,81795642181faf3f
Normal-5709.png,Normal,Normal/images/Normal-5709.png,Normal/masks/Normal-5709.png,True,G14839,train,868ee7e11edb5e076092ff3974a25afc,993986421f7d6a39
Normal-571.png,Normal,Normal/images/Normal-571.png,Normal/masks/Normal-571.png,True,G14840,train,3bad402a3e87e9c3a8f779c26a62ae36,917fcf3a090d19c5
Normal-5710.png,Normal,Normal/images/Normal-5710.png,Normal/masks/Normal-5710.png,True,G14841,val,78f690fd9e369ff5792b32bf8fdd18cd,cc2fc2531d414e3f
Normal-5711.png,Normal,Normal/images/Normal-5711.png,Normal/masks/Normal-5711.png,True,G14842,val,f519079ec64d107a54866f6ac7e8fee0,cc3cc3431bc24b9f
Normal-5712.png,Normal,Normal/images/Normal-5712.png,Normal/masks/Normal-5712.png,True,G14843,val,bd5c34ab03fffa4cb2b41f56893f3ab5,d42f0a5d5c4e38d6
Normal-5713.png,Normal,Normal/images/Normal-5713.png,Normal/masks/Normal-5713.png,True,G14844,test,2a9fdeda6668d724f3f90c7ddb1a2404,c93ac21fc8adbaa4
Normal-5714.png,Normal,Normal/images/Normal-5714.png,Normal/masks/Normal-5714.png,True,G14845,train,583aea4a1c160ed1890f35995e25a384,d96ec64b094f0f19
Normal-5715.png,Normal,Normal/images/Normal-5715.png,Normal/masks/Normal-5715.png,True,G14846,train,0619f00b7cb178bbd53d66b7805b2473,89f9871a855f3a45
Normal-5716.png,Normal,Normal/images/Normal-5716.png,Normal/masks/Normal-5716.png,True,G14847,train,5a82e60f313af14fb55a67320467c138,c93d86431f4d4b1e
Normal-5717.png,Normal,Normal/images/Normal-5717.png,Normal/masks/Normal-5717.png,True,G14848,test,d463d0dd83e478a6edf373065d2008f2,987fc456196d4479
Normal-5718.png,Normal,Normal/images/Normal-5718.png,Normal/masks/Normal-5718.png,True,G14849,train,9484f376fe3306faeb244c68584fed19,dd3d820f98c61f91
Normal-5719.png,Normal,Normal/images/Normal-5719.png,Normal/masks/Normal-5719.png,True,G14850,train,cde41ecb875852644b984a511d72890f,9929c616b85bad78
Normal-572.png,Normal,Normal/images/Normal-572.png,Normal/masks/Normal-572.png,True,G14851,train,a1839fcc90b4999abb713e3814c11b56,d47ec31bcb8568c4
Normal-5720.png,Normal,Normal/images/Normal-5720.png,Normal/masks/Normal-5720.png,True,G14852,train,c9a8d0753672156c183a421b14b642b1,887f475a55465e51
Normal-5721.png,Normal,Normal/images/Normal-5721.png,Normal/masks/Normal-5721.png,True,G14853,train,a4b4b1604e9602807370745ca4cd9d02,dc3dc30f490ebc2c
Normal-5722.png,Normal,Normal/images/Normal-5722.png,Normal/masks/Normal-5722.png,True,G14854,val,00c089ce46fc1b5a0726e7dc825af765,993d061a7b6d38e1
Normal-5723.png,Normal,Normal/images/Normal-5723.png,Normal/masks/Normal-5723.png,True,G14855,train,2497a1f590b6c47a4a31b661419aa62c,c17c424d1e4f3d3c
Normal-5724.png,Normal,Normal/images/Normal-5724.png,Normal/masks/Normal-5724.png,True,G14856,train,29d7f3367cca09cb301b97ddbf3a402c,dc2ed2534bc24b0f
Normal-5725.png,Normal,Normal/images/Normal-5725.png,Normal/masks/Normal-5725.png,True,G14857,train,58d670d6cca68867dc0ce1712b7c980d,dd7cc319410f0b4f
Normal-5726.png,Normal,Normal/images/Normal-5726.png,Normal/masks/Normal-5726.png,True,G14858,train,e3048025246730837568ca180ef6aba1,803f421f5b4f3c2d
Normal-5727.png,Normal,Normal/images/Normal-5727.png,Normal/masks/Normal-5727.png,True,G14859,val,f45d25792f30ae35c0f418cf0e6ad2ce,993f461b081b1bee
Normal-5728.png,Normal,Normal/images/Normal-5728.png,Normal/masks/Normal-5728.png,True,G14860,train,0012f61a66b99b316e5bda792e1c72ab,cc3dd2431cc46f1e
Normal-5729.png,Normal,Normal/images/Normal-5729.png,Normal/masks/Normal-5729.png,True,G14861,test,ba166bc30176950df27e620430ece8f8,8979071a5b557879
Normal-573.png,Normal,Normal/images/Normal-573.png,Normal/masks/Normal-573.png,True,G14862,train,24a8ffaf481c142d7d6713bce87056d6,9129c71edecc68d4
Normal-5730.png,Normal,Normal/images/Normal-5730.png,Normal/masks/Normal-5730.png,True,G14863,test,fd789016be5433797449484639d5e70c,d17b461a196f2939
Normal-5731.png,Normal,Normal/images/Normal-5731.png,Normal/masks/Normal-5731.png,True,G14864,train,f1d0c9ccdfed206bf3124a4eed91793c,996806185bed78f9
Normal-5732.png,Normal,Normal/images/Normal-5732.png,Normal/masks/Normal-5732.png,True,G14865,val,a2e6294404590897978c3015a8c0493f,dc2cd21fc64b5549
Normal-5733.png,Normal,Normal/images/Normal-5733.png,Normal/masks/Normal-5733.png,True,G14866,train,c735eaf6de785f351429f0a70abc26d9,89785718195d395f
Normal-5734.png,Normal,Normal/images/Normal-5734.png,Normal/masks/Normal-5734.png,True,G14867,train,ad2324fedff1db36a723c49634d9f4c1,91394e07194fed3a
Normal-5735.png,Normal,Normal/images/Normal-5735.png,Normal/masks/Normal-5735.png,True,G14868,train,82fb3e92e4b30bad993d37aede5ace6a,d9fcc65bc45a504a
Normal-5736.png,Normal,Normal/images/Normal-5736.png,Normal/masks/Normal-5736.png,True,G14869,train,2466cb413e0a180471c51302fdd1a32d,f9598616181e1f3b
Normal-5737.png,Normal,Normal/images/Normal-5737.png,Normal/masks/Normal-5737.png,True,G14870,test,33f51ab5eaa58af1bc73e91d0113534e,9169161e1e9e3ab9
Normal-5738.png,Normal,Normal/images/Normal-5738.png,Normal/masks/Normal-5738.png,True,G14871,train,6f8fd1e839801202add1f3fb47885a26,dd2f860b194d4f19
Normal-5739.png,Normal,Normal/images/Normal-5739.png,Normal/masks/Normal-5739.png,True,G14872,train,bb560e2a26ccadebf4411e6fbbb89d60,907c57d30a50cf5e
Normal-574.png,Normal,Normal/images/Normal-574.png,Normal/masks/Normal-574.png,True,G14873,train,33803a3c34e0765fa648d116b01fcf2f,d12dc60bd86979b8
Normal-5740.png,Normal,Normal/images/Normal-5740.png,Normal/masks/Normal-5740.png,True,G14874,train,73106efc765e670cc958cfc4360cae63,b969861e964e3a55
Normal-5741.png,Normal,Normal/images/Normal-5741.png,Normal/masks/Normal-5741.png,True,G14875,train,8301187d27ab3835c9eb064f6c0f4acb,d92f061e4e5a5c69
Normal-5742.png,Normal,Normal/images/Normal-5742.png,Normal/masks/Normal-5742.png,True,G14876,train,3e2e373fccdcae8fdfb92b814dbd8c92,997f061b454f114f
Normal-5743.png,Normal,Normal/images/Normal-5743.png,Normal/masks/Normal-5743.png,True,G14877,train,277464b327bbb8491caa4f69828885e2,997f861a055a714f
Normal-5744.png,Normal,Normal/images/Normal-5744.png,Normal/masks/Normal-5744.png,True,G14878,train,503acf7ab5fcdd9cf5d3ecc4eac6d16f,d97dc4521a555a1e
Normal-5745.png,Normal,Normal/images/Normal-5745.png,Normal/masks/Normal-5745.png,True,G14879,train,7b807254cd50fb37521e28b49df5cadd,817906565e4e5f59
Normal-5746.png,Normal,Normal/images/Normal-5746.png,Normal/masks/Normal-5746.png,True,G14880,val,ca17c710aa0468390575ffef459d6797,9979061a154f39f9
Normal-5747.png,Normal,Normal/images/Normal-5747.png,Normal/masks/Normal-5747.png,True,G14881,val,fd5040377a9b40d10b09b39aa40a31b4,995d465a085f187f
Normal-5748.png,Normal,Normal/images/Normal-5748.png,Normal/masks/Normal-5748.png,True,G14882,val,46bd804be9eb6183083f7b671ffd67b7,c43c424b2d2ff47c
Normal-5749.png,Normal,Normal/images/Normal-5749.png,Normal/masks/Normal-5749.png,True,G14883,test,827c0079ce8e892a29786b5c13139296,88fdc61f92471a59
Normal-575.png,Normal,Normal/images/Normal-575.png,Normal/masks/Normal-575.png,True,G14884,test,ce46eeb37086182e1f53426db9305acb,98fdc31e8b0d3c85
Normal-5750.png,Normal,Normal/images/Normal-5750.png,Normal/masks/Normal-5750.png,True,G14885,test,ed1a5c3448a772dac3824e85d1ed4bdf,d03d431f1c0cef39
Normal-5751.png,Normal,Normal/images/Normal-5751.png,Normal/masks/Normal-5751.png,True,G14886,train,f390e0a730ee1595a9b2c9f9d3ffe9cb,986dc7171cc55b58
Normal-5752.png,Normal,Normal/images/Normal-5752.png,Normal/masks/Normal-5752.png,True,G14887,train,2af55016974000b8dcc28e584756652c,d52d86161b4f1a5b
Normal-5753.png,Normal,Normal/images/Normal-5753.png,Normal/masks/Normal-5753.png,True,G14888,train,5b0d3a3555d58defddbb4dc22b5dafa0,d92bd619182b3d87
Normal-5754.png,Normal,Normal/images/Normal-5754.png,Normal/masks/Normal-5754.png,True,G14889,train,e8a5d27147db0f481f5d70167fe2682b,e865561b471b4d9a
Normal-5755.png,Normal,Normal/images/Normal-5755.png,Normal/masks/Normal-5755.png,True,G14890,train,261e2986c67c890fca1b5e3a093846cc,dd7cd65b414e1a11
Normal-5756.png,Normal,Normal/images/Normal-5756.png,Normal/masks/Normal-5756.png,True,G14891,train,7835fbdaaf59315dd513b8d77971922d,8878071e1f5f3a59
Normal-5757.png,Normal,Normal/images/Normal-5757.png,Normal/masks/Normal-5757.png,True,G14892,train,d77b37ff57c71700014f226d2f932fa6,997985171a954f5a
Normal-5758.png,Normal,Normal/images/Normal-5758.png,Normal/masks/Normal-5758.png,True,G14893,train,4fdf1369f7885191d126dd1701af747c,91695e1f7a4d6838
Normal-5759.png,Normal,Normal/images/Normal-5759.png,Normal/masks/Normal-5759.png,True,G14894,train,76bfa797f26b54894ee797a664215341,997b861e1d485b39
Normal-576.png,Normal,Normal/images/Normal-576.png,Normal/masks/Normal-576.png,True,G14895,train,3362982f4ea895f39c43b0f6d86ac423,91fc9e388978b84d
Normal-5760.png,Normal,Normal/images/Normal-5760.png,Normal/masks/Normal-5760.png,True,G14896,train,35ecfa74eda2eb00722df7892de94cfc,dd38521f524f1d4a
Normal-5761.png,Normal,Normal/images/Normal-5761.png,Normal/masks/Normal-5761.png,True,G14897,train,f23b821ffaf9d0c47a7d1e623dd8e65f,dd2a431d0e1bb8ae
Normal-5762.png,Normal,Normal/images/Normal-5762.png,Normal/masks/Normal-5762.png,True,G14898,train,ef139257f904781dd4e9d3e40c5b26d1,81f91e0f111e5f1b
Normal-5763.png,Normal,Normal/images/Normal-5763.png,Normal/masks/Normal-5763.png,True,G14899,val,00513aec922fe37174deee244db9765c,9979c7161e445b1d
Normal-5764.png,Normal,Normal/images/Normal-5764.png,Normal/masks/Normal-5764.png,True,G14900,val,097dec04d856e686d02e3c1a31468e91,916a7e9b61849a7a
Normal-5765.png,Normal,Normal/images/Normal-5765.png,Normal/masks/Normal-5765.png,True,G14901,test,edf7e615b5efd17a93becd4801af2e2c,dd0cc71b1d4c0f1e
Normal-5766.png,Normal,Normal/images/Normal-5766.png,Normal/masks/Normal-5766.png,True,G14902,val,19ffe0103e2cefe0110ea80ce9292ecc,ac2ac79a17861e1f
Normal-5767.png,Normal,Normal/images/Normal-5767.png,Normal/masks/Normal-5767.png,True,G14903,test,24ddc8549fab24c15d54f7be0c4b800b,d93bc50619594f1d
Normal-5768.png,Normal,Normal/images/Normal-5768.png,Normal/masks/Normal-5768.png,True,G14904,test,5ed38471300d2c33af8d4f3c4cade2f6,cc3dc354524f0d4f
Normal-5769.png,Normal,Normal/images/Normal-5769.png,Normal/masks/Normal-5769.png,True,G14905,train,25306a32d0879ee1006b3ae28a7b9c88,d93d061a194c7bb9
Normal-577.png,Normal,Normal/images/Normal-577.png,Normal/masks/Normal-577.png,True,G14906,train,5652368e08ec55364823e9616e801bd5,913bc61a98d939d5
Normal-5770.png,Normal,Normal/images/Normal-5770.png,Normal/masks/Normal-5770.png,True,G14907,train,58b010c1d81cea399559a600f7ce021c,dc3c421b0d2e3fb4
Normal-5771.png,Normal,Normal/images/Normal-5771.png,Normal/masks/Normal-5771.png,True,G14908,train,ef120917f97f0525a53667830bcb4500,817886161b5d4f7b
Normal-5772.png,Normal,Normal/images/Normal-5772.png,Normal/masks/Normal-5772.png,True,G14909,train,2cc290512ef47c17c274f688ebd08cf8,d97b561a192939b8
Normal-5773.png,Normal,Normal/images/Normal-5773.png,Normal/masks/Normal-5773.png,True,G14910,train,dbfb2e7ae95dc2fe41d691f6cdc807f9,cc3d420f0e0e2ffc
Normal-5774.png,Normal,Normal/images/Normal-5774.png,Normal/masks/Normal-5774.png,True,G14911,train,541b17299a8f3385dccff8077792e2e3,c8fac71bd44a1c4d
Normal-5775.png,Normal,Normal/images/Normal-5775.png,Normal/masks/Normal-5775.png,True,G14912,train,9a411a43b3d06cd337d621b00ca23055,d17f471b174f1a08
Normal-5776.png,Normal,Normal/images/Normal-5776.png,Normal/masks/Normal-5776.png,True,G14913,train,237fc1d705e71e5373a2b9b4b2e6d851,dd2ec25f41471b51
Normal-5777.png,Normal,Normal/images/Normal-5777.png,Normal/masks/Normal-5777.png,True,G14914,train,b8ae228b013e4d5e0192861c410712cb,887907165e0d7f1b
Normal-5778.png,Normal,Normal/images/Normal-5778.png,Normal/masks/Normal-5778.png,True,G14915,val,ccc4f5c0d456ea4321d12515cbd1f4dc,d87b56861a0faa1b
Normal-5779.png,Normal,Normal/images/Normal-5779.png,Normal/masks/Normal-5779.png,True,G14916,train,3552ae72bd08c6f925990a9663c4aab2,9979c71516071e97
Normal-578.png,Normal,Normal/images/Normal-578.png,Normal/masks/Normal-578.png,True,G14917,train,cf1af6fabb955cf1a7c360a43e83cc3f,d57f871e890d2887
Normal-5780.png,Normal,Normal/images/Normal-5780.png,Normal/masks/Normal-5780.png,True,G14918,train,70ec1456bd8f799bc0fa30008c555216,ec5cd35b4d481c96
Normal-5781.png,Normal,Normal/images/Normal-5781.png,Normal/masks/Normal-5781.png,True,G14919,train,5549d3a1bad31454598c1c0f76ea89f4,d43d83431c6cef29
Normal-5782.png,Normal,Normal/images/Normal-5782.png,Normal/masks/Normal-5782.png,True,G14920,train,2d6e7f26e4ee5d99ff1788c45d944cf3,d93d46431f1c6b49
Normal-5783.png,Normal,Normal/images/Normal-5783.png,Normal/masks/Normal-5783.png,True,G14921,val,737891a6a2b8bca0bc392d6133dde0eb,dd2dc61f484a3b89
Normal-5784.png,Normal,Normal/images/Normal-5784.png,Normal/masks/Normal-5784.png,True,G14922,test,b334046a04b8a975c84ae1a4ed08b5a4,91798616185f7e3a
Normal-5785.png,Normal,Normal/images/Normal-5785.png,Normal/masks/Normal-5785.png,True,G14923,val,709fc6f3d26bf69e798b8dd9e513b1b8,9179861e1a5f785a
Normal-5786.png,Normal,Normal/images/Normal-5786.png,Normal/masks/Normal-5786.png,True,G14924,test,0e4252298ec06f161aa0fd20cf5fd744,8c4f074b496e4aeb
Normal-5787.png,Normal,Normal/images/Normal-5787.png,Normal/masks/Normal-5787.png,True,G14925,train,2d66bfa66da8920d33dfceb7359e3119,9979861a194d7ba9
Normal-5788.png,Normal,Normal/images/Normal-5788.png,Normal/masks/Normal-5788.png,True,G14926,test,b5dfd98623e2ceacf108e620a8ce6dc0,91288617b8d7e8f9
Normal-5789.png,Normal,Normal/images/Normal-5789.png,Normal/masks/Normal-5789.png,True,G14927,train,6b70505ff38df5751c13160a348d161a,d939061e454f39e5
Normal-579.png,Normal,Normal/images/Normal-579.png,Normal/masks/Normal-579.png,True,G14928,train,19ea03a145439780461323a313b3df26,d42dc30fed6c0ce4
Normal-5790.png,Normal,Normal/images/Normal-5790.png,Normal/masks/Normal-5790.png,True,G14929,train,217f579bf34d49207db2f595934c82d0,9979c51616c74e1e
Normal-5791.png,Normal,Normal/images/Normal-5791.png,Normal/masks/Normal-5791.png,True,G14930,train,c872eeca2898420292c8cb5a09ee831c,f979071e1a097b38
Normal-5792.png,Normal,Normal/images/Normal-5792.png,Normal/masks/Normal-5792.png,True,G14931,val,34e05397996bc7fc92deb6e8b05f47f7,d43dc24b0f4a2f1e
Normal-5793.png,Normal,Normal/images/Normal-5793.png,Normal/masks/Normal-5793.png,True,G14932,val,7ac3c1cd36c7808955923b57da2d4733,997b070a1f49693d
Normal-5794.png,Normal,Normal/images/Normal-5794.png,Normal/masks/Normal-5794.png,True,G14933,train,6b541f8e16c71ce8b5e4c765154ad2c1,dd2d42571b426f1a
Normal-5795.png,Normal,Normal/images/Normal-5795.png,Normal/masks/Normal-5795.png,True,G14934,train,f0a507137f9c884bc5d7725fe418cfad,917d46121b49ff19
Normal-5796.png,Normal,Normal/images/Normal-5796.png,Normal/masks/Normal-5796.png,True,G14935,train,b23f76aa5ff7b9eb5ac9457b2e2f5873,c43d42431bcaadf9
Normal-5797.png,Normal,Normal/images/Normal-5797.png,Normal/masks/Normal-5797.png,True,G14936,train,5faa563defea490e5130cb5d0ef5a880,9169c616975e3e1a
Normal-5798.png,Normal,Normal/images/Normal-5798.png,Normal/masks/Normal-5798.png,True,G14937,train,3f8409955e713a9c3a38f6c297941324,dc2ec34b0c0e7e3c
Normal-5799.png,Normal,Normal/images/Normal-5799.png,Normal/masks/Normal-5799.png,True,G14938,train,0a51cc217f7832639fba989c899fc0e7,9979865a59457a39
Normal-58.png,Normal,Normal/images/Normal-58.png,Normal/masks/Normal-58.png,True,G14939,test,709aa725da31b5e94d4d6290e3b0af20,d17f5618194d391d
Normal-580.png,Normal,Normal/images/Normal-580.png,Normal/masks/Normal-580.png,True,G14940,train,8114b118a5165d3e0386ff875ceb6eb5,d06ec71acd6d01c7
Normal-5800.png,Normal,Normal/images/Normal-5800.png,Normal/masks/Normal-5800.png,True,G14941,train,f451d57fa176114cd54b20fb9783d183,dd0842c30eed2bf5
Normal-5801.png,Normal,Normal/images/Normal-5801.png,Normal/masks/Normal-5801.png,True,G14942,train,f1dc1805acc680def63c93ce0c35f032,dc3d430b1acf2f18
Normal-5802.png,Normal,Normal/images/Normal-5802.png,Normal/masks/Normal-5802.png,True,G14943,val,a9ec1a069e72924aa2dc569d0bf36b45,d929841b0d4f3f69
Normal-5803.png,Normal,Normal/images/Normal-5803.png,Normal/masks/Normal-5803.png,True,G14944,test,ca275459e0af716dd31bd19b7950d528,892e06421f575f5e
Normal-5804.png,Normal,Normal/images/Normal-5804.png,Normal/masks/Normal-5804.png,True,G14945,test,a45104c9656671b9446adbbbb39867d6,817f049a1dd85fcc
Normal-5805.png,Normal,Normal/images/Normal-5805.png,Normal/masks/Normal-5805.png,True,G14946,train,d36f1b29cfd8572ae62fa6c9fee9c47a,9968c61b1bee78a4
Normal-5806.png,Normal,Normal/images/Normal-5806.png,Normal/masks/Normal-5806.png,True,G14947,train,c58c12c1524272d1ed29e265ac7ef0e2,913b161e1a7b3a39
Normal-5807.png,Normal,Normal/images/Normal-5807.png,Normal/masks/Normal-5807.png,True,G14948,train,00dcbaa11cc29242758d0c1c85a5937c,d07d4a525d4b4f1c
Normal-5808.png,Normal,Normal/images/Normal-5808.png,Normal/masks/Normal-5808.png,True,G14949,train,82b5ea9df60d351a0f13abff0191482b,817f061a5f0d2a7d
Normal-5809.png,Normal,Normal/images/Normal-5809.png,Normal/masks/Normal-5809.png,True,G14950,train,759b32c3a1899bbc96e965538b795191,993d061e1a4d5a5f
Normal-581.png,Normal,Normal/images/Normal-581.png,Normal/masks/Normal-581.png,True,G14951,train,559eded84de61825eeb165c9c0ef673a,91afc61ad87911c7
Normal-5810.png,Normal,Normal/images/Normal-5810.png,Normal/masks/Normal-5810.png,True,G14952,train,552523539d557747cbcaf9b21addf4de,d43cc7195c4a396d
Normal-5811.png,Normal,Normal/images/Normal-5811.png,Normal/masks/Normal-5811.png,True,G14953,train,32411858938bd329879f00961ffb67ea,917957861a7de838
Normal-5812.png,Normal,Normal/images/Normal-5812.png,Normal/masks/Normal-5812.png,True,G14954,train,17a9d9896b6bbb223d3736151cd1ea96,9979061e5b065eec
Normal-5813.png,Normal,Normal/images/Normal-5813.png,Normal/masks/Normal-5813.png,True,G14955,train,5c07947fed5aab0e61a95c3d526322eb,9d201e5e68ee69c5
Normal-5814.png,Normal,Normal/images/Normal-5814.png,Normal/masks/Normal-5814.png,True,G14956,train,2c0f8366dd8e8b0870d0ef3db3b67ffd,dd3dc61211c61f93
Normal-5815.png,Normal,Normal/images/Normal-5815.png,Normal/masks/Normal-5815.png,True,G14957,val,60a9857d617d609beb0a485b7f0d2b46,c13a561d1e4f3bc4
Normal-5816.png,Normal,Normal/images/Normal-5816.png,Normal/masks/Normal-5816.png,True,G14958,train,ca5ee79c005f0f58a87ef987b9e183de,917a04171f1c5b5f
Normal-5817.png,Normal,Normal/images/Normal-5817.png,Normal/masks/Normal-5817.png,True,G14959,val,780ca734eeed9c213cbe7be28a50246b,dc3d425b0b4a3c7a
Normal-5818.png,Normal,Normal/images/Normal-5818.png,Normal/masks/Normal-5818.png,True,G14960,test,5c38dd973427e2147fc479caa1ed6513,95e90c0e1b1f5c79
Normal-5819.png,Normal,Normal/images/Normal-5819.png,Normal/masks/Normal-5819.png,True,G14961,train,2ef16c559204ae5ffb70674b4eb25617,897987161a5f3e58
Normal-582.png,Normal,Normal/images/Normal-582.png,Normal/masks/Normal-582.png,True,G14962,train,f9a312e5a1ede56a3d4b95992d64d157,d12dc65a584d5d5c
Normal-5820.png,Normal,Normal/images/Normal-5820.png,Normal/masks/Normal-5820.png,True,G14963,train,91bede3a651fdbb0fba523b310929a7a,dc29534f40697e78
Normal-5821.png,Normal,Normal/images/Normal-5821.png,Normal/masks/Normal-5821.png,True,G14964,val,6a17949bb11a5e5f4f8a563820991b76,d93e864b1e4c4d3a
Normal-5822.png,Normal,Normal/images/Normal-5822.png,Normal/masks/Normal-5822.png,True,G14965,test,e23fe6de0728238e1869fbd00412155b,d93dc61a154a1b1f
Normal-5823.png,Normal,Normal/images/Normal-5823.png,Normal/masks/Normal-5823.png,True,G14966,test,208ac1ad50f2d12fbc12b9120c4a783b,dc6ec35b484b0a4f
Normal-5824.png,Normal,Normal/images/Normal-5824.png,Normal/masks/Normal-5824.png,True,G14967,train,b8cbc13a968388855d4191908f9ced40,d4af434d4e0f0e4b
Normal-5825.png,Normal,Normal/images/Normal-5825.png,Normal/masks/Normal-5825.png,True,G14968,train,f1d07612d6f4d3cb8f94c321c26b2611,9b7a85145ae85cf8
Normal-5826.png,Normal,Normal/images/Normal-5826.png,Normal/masks/Normal-5826.png,True,G14969,train,9a9a469d796a3b4e01e0b049ca25754b,817a861a19cfbab9
Normal-5827.png,Normal,Normal/images/Normal-5827.png,Normal/masks/Normal-5827.png,True,G14970,train,d238620b3b18ab5eb830e39a321d68b1,d97846191c6f39e1
Normal-5828.png,Normal,Normal/images/Normal-5828.png,Normal/masks/Normal-5828.png,True,G14971,train,b2a943cf76ee8f16ec893bbd68ba9018,d97bc6184959584f
Normal-5829.png,Normal,Normal/images/Normal-5829.png,Normal/masks/Normal-5829.png,True,G14972,train,c9fb8e581198842633526a7af717e92c,c13dc61e1a4a6d3d
Normal-583.png,Normal,Normal/images/Normal-583.png,Normal/masks/Normal-583.png,True,G14973,train,32c57bbd4d007a528f226f23e9351a36,d07ec65b9d4c2c94
Normal-5830.png,Normal,Normal/images/Normal-5830.png,Normal/masks/Normal-5830.png,True,G14974,train,e4f7681cd840b37236866bac9b62760c,817887961a9785fb
Normal-5831.png,Normal,Normal/images/Normal-5831.png,Normal/masks/Normal-5831.png,True,G14975,train,885336d5189eaf98c0c24e14977399e6,9c64530f551f554e
Normal-5832.png,Normal,Normal/images/Normal-5832.png,Normal/masks/Normal-5832.png,True,G14976,train,349e3f23a46102c9d87d682487dee0b4,dc3cc25b0d4a1dbc
Normal-5833.png,Normal,Normal/images/Normal-5833.png,Normal/masks/Normal-5833.png,True,G14977,train,8f23e5b93318f255e47afce301443b77,cc3dc20d124e3dfc
Normal-5834.png,Normal,Normal/images/Normal-5834.png,Normal/masks/Normal-5834.png,True,G14978,train,f782e9d411bbacac34abec1cbd543f90,dd3d424f1f08ad38
Normal-5835.png,Normal,Normal/images/Normal-5835.png,Normal/masks/Normal-5835.png,True,G14979,train,baae184f178bc53f3d1674f42e058fa7,b925c61b451f5c4d
Normal-5836.png,Normal,Normal/images/Normal-5836.png,Normal/masks/Normal-5836.png,True,G14980,val,e4cea584367b4dc3dff1963b2cc0164f,817d86125b5f7859
Normal-5837.png,Normal,Normal/images/Normal-5837.png,Normal/masks/Normal-5837.png,True,G14981,val,d7c46a17c5ae53144a8c4daf232e9c70,880f065a5d6d78ed
Normal-5838.png,Normal,Normal/images/Normal-5838.png,Normal/masks/Normal-5838.png,True,G14982,val,479984c147552e30668182e6dda141cb,91f96a865a18576d
Normal-5839.png,Normal,Normal/images/Normal-5839.png,Normal/masks/Normal-5839.png,True,G14983,test,4d843e77bf870d01b533af733526894d,9968861a1f1b1f5e
Normal-584.png,Normal,Normal/images/Normal-584.png,Normal/masks/Normal-584.png,True,G14984,train,2d3d807330a8add2f023f8e1374e9ae7,9968c51acb4f584f
Normal-5840.png,Normal,Normal/images/Normal-5840.png,Normal/masks/Normal-5840.png,True,G14985,test,e28612352cb45b2a4617360d05448245,9925061f071f175f
Normal-5841.png,Normal,Normal/images/Normal-5841.png,Normal/masks/Normal-5841.png,True,G14986,test,d6b521099d864905bb6e5f1b8947dba4,9905061f4d1d4f7e
Normal-5842.png,Normal,Normal/images/Normal-5842.png,Normal/masks/Normal-5842.png,True,G14987,train,451872200268fc4847bb4ee7267d6aa9,997b05169a055b7b
Normal-5843.png,Normal,Normal/images/Normal-5843.png,Normal/masks/Normal-5843.png,True,G14988,train,b23fec4765a6805905f5a69855ad1a32,817b87861a195d5f
Normal-5844.png,Normal,Normal/images/Normal-5844.png,Normal/masks/Normal-5844.png,True,G14989,train,9c631669c627f6d37b3747d434fa6ec6,d979c7161b544a1d
Normal-5845.png,Normal,Normal/images/Normal-5845.png,Normal/masks/Normal-5845.png,True,G14990,val,4cc1017d6637d92ecb57a041980407d3,d03d43535f0a4d1f
Normal-5846.png,Normal,Normal/images/Normal-5846.png,Normal/masks/Normal-5846.png,True,G14991,train,ffd8a9d6d7f2ba5d545a2d7ee89614c4,9f6158181b39ede8
Normal-5847.png,Normal,Normal/images/Normal-5847.png,Normal/masks/Normal-5847.png,True,G14992,train,ea6478f6d4ca717030c1fa39a09be753,997d871a015f394d
Normal-5848.png,Normal,Normal/images/Normal-5848.png,Normal/masks/Normal-5848.png,True,G14993,train,484db14b615b33b5020a4f85dbf4e57f,d97a87421e4d5b1c
Normal-5849.png,Normal,Normal/images/Normal-5849.png,Normal/masks/Normal-5849.png,True,G14994,val,c87797dcfd365e4d9aa7e3c707f42217,9d2d861a9cc6bab8
Normal-585.png,Normal,Normal/images/Normal-585.png,Normal/masks/Normal-585.png,True,G14995,train,27fdf0b63586b43a1daa80699bc13109,8875a73aa93d6c2c
Normal-5850.png,Normal,Normal/images/Normal-5850.png,Normal/masks/Normal-5850.png,True,G14996,train,b7191803b76bc92820f28f4ed54ceeee,997fc61a194939e8
Normal-5851.png,Normal,Normal/images/Normal-5851.png,Normal/masks/Normal-5851.png,True,G14997,train,4a442148768262a59be56c0b924db900,997f061a591a5f4a
Normal-5852.png,Normal,Normal/images/Normal-5852.png,Normal/masks/Normal-5852.png,True,G14998,train,cdbe19e37ee6c641018cff368043c17b,8959045e185f5b7b
Normal-5853.png,Normal,Normal/images/Normal-5853.png,Normal/masks/Normal-5853.png,True,G14999,train,c62d3a2c8948ed7d040737e2c57bca07,993dc20f1aca6bb8
Normal-5854.png,Normal,Normal/images/Normal-5854.png,Normal/masks/Normal-5854.png,True,G15000,val,05842f64625a13f27261c74aaf41462b,887987165a5f784b
Normal-5855.png,Normal,Normal/images/Normal-5855.png,Normal/masks/Normal-5855.png,True,G15001,train,4c36ea3902a1d351ecaa875da95583b8,dd2c50035b574b1f
Normal-5856.png,Normal,Normal/images/Normal-5856.png,Normal/masks/Normal-5856.png,True,G15002,train,9d6360b89fad22538d803a56bd856900,887b07861e9a7fd0
Normal-5857.png,Normal,Normal/images/Normal-5857.png,Normal/masks/Normal-5857.png,True,G15003,train,bdc99cba0a4af5d6eebc9f8e409c3dc9,dd2cd2131dc60f3c
Normal-5858.png,Normal,Normal/images/Normal-5858.png,Normal/masks/Normal-5858.png,True,G15004,train,91e3a3568bd15f4b3be8b48985e3e839,dd2ac25f1f0a4b1c
Normal-5859.png,Normal,Normal/images/Normal-5859.png,Normal/masks/Normal-5859.png,True,G15005,test,d5a9ee78c5e531f140e5a27649c4ed1c,997b861a1a4d5b39
Normal-586.png,Normal,Normal/images/Normal-586.png,Normal/masks/Normal-586.png,True,G15006,test,9d7ef92110b4f0bc5d436acd5054ca4d,9129e63af8693971
Normal-5860.png,Normal,Normal/images/Normal-5860.png,Normal/masks/Normal-5860.png,True,G15007,test,ba5ce0629f7155a274a0253e9aea63b2,d12f461f1c461fd4
Normal-5861.png,Normal,Normal/images/Normal-5861.png,Normal/masks/Normal-5861.png,True,G15008,train,331d04c807dca544abb2b959743a6667,8378c4167a15fa5b
Normal-5862.png,Normal,Normal/images/Normal-5862.png,Normal/masks/Normal-5862.png,True,G15009,train,e83c4b7125e738738fb68828c9b250cb,d47e42531b43075f
Normal-5863.png,Normal,Normal/images/Normal-5863.png,Normal/masks/Normal-5863.png,True,G15010,train,3af494a24909bde530ed7be4d547f8f9,dc28c20fc3575f49
Normal-5864.png,Normal,Normal/images/Normal-5864.png,Normal/masks/Normal-5864.png,True,G15011,train,af0742c91a79e50160e61fd191f7e6cd,d82dc45a6d7838e5
Normal-5865.png,Normal,Normal/images/Normal-5865.png,Normal/masks/Normal-5865.png,True,G15012,train,db01e2d2049080bfbb01f34d27c61a3e,897a57192e6b388d
Normal-5866.png,Normal,Normal/images/Normal-5866.png,Normal/masks/Normal-5866.png,True,G15013,train,164d341942d7c6683ec8b29d7ba15779,887d061e1a5f1f1d
Normal-5867.png,Normal,Normal/images/Normal-5867.png,Normal/masks/Normal-5867.png,True,G15014,train,79c3c9c69afbbc0cbd22a5db84447b5b,8179d606181ffd1b
Normal-5868.png,Normal,Normal/images/Normal-5868.png,Normal/masks/Normal-5868.png,True,G15015,train,2974f6241c806ff30e4091d9529b3dc2,8879c71e1d4f1b29
Normal-5869.png,Normal,Normal/images/Normal-5869.png,Normal/masks/Normal-5869.png,True,G15016,train,a143655912e500c82ff2a3bd3c2f0cc9,d42e535f0a4f1c4e
Normal-587.png,Normal,Normal/images/Normal-587.png,Normal/masks/Normal-587.png,True,G15017,train,a526f5d55094a5cbc4b9aab3e2fa8bb3,d46f8f1a484748d7
Normal-5870.png,Normal,Normal/images/Normal-5870.png,Normal/masks/Normal-5870.png,True,G15018,train,b8a0daf1b579b6aaa693fcfa5e1f0d13,d939c4131fc51a9e
Normal-5871.png,Normal,Normal/images/Normal-5871.png,Normal/masks/Normal-5871.png,True,G15019,train,4a6ba0258d33d3866b011b088661c5d9,d92942075d0aff39
Normal-5872.png,Normal,Normal/images/Normal-5872.png,Normal/masks/Normal-5872.png,True,G15020,val,76d94c3b6f933cfd3898d91232c7854f,c1b96ed61a902f4b
Normal-5873.png,Normal,Normal/images/Normal-5873.png,Normal/masks/Normal-5873.png,True,G15021,val,914490d720bc1e8189fb9cef9177c6ef,dd2852c75b154d4b
Normal-5874.png,Normal,Normal/images/Normal-5874.png,Normal/masks/Normal-5874.png,True,G15022,test,4c074b90c11f5976220d6917825af63b,d03db6c25b19d04f
Normal-5875.png,Normal,Normal/images/Normal-5875.png,Normal/masks/Normal-5875.png,True,G15023,val,6a87fb564dae5e7f1b932cfeac220666,c83b57871e150eda
Normal-5876.png,Normal,Normal/images/Normal-5876.png,Normal/masks/Normal-5876.png,True,G15024,test,18f4b80fbb9381feb0f74224e4fd34e8,dd2cd25bd10399d1
Normal-5877.png,Normal,Normal/images/Normal-5877.png,Normal/masks/Normal-5877.png,True,G15025,test,cb422b355d280abd86ade0691320dd73,dd2e825f984d2e54
Normal-5878.png,Normal,Normal/images/Normal-5878.png,Normal/masks/Normal-5878.png,True,G15026,train,a235d918d1ba4df3b65d38af9d94e21e,895f965a59e9a8a1
Normal-5879.png,Normal,Normal/images/Normal-5879.png,Normal/masks/Normal-5879.png,True,G15027,train,5aaaddee1cbbc0bb1f8cc807a9df0503,d47c43c24b0bc74f
Normal-588.png,Normal,Normal/images/Normal-588.png,Normal/masks/Normal-588.png,True,G15028,train,3a62a0bf50ed0fab1d99c3c1829b81fe,81bec618c85b9bc7
Normal-5880.png,Normal,Normal/images/Normal-5880.png,Normal/masks/Normal-5880.png,True,G15029,train,8e71a21db452a6a02d475b7fd3db6867,d57f06421b1c687b
Normal-5881.png,Normal,Normal/images/Normal-5881.png,Normal/masks/Normal-5881.png,True,G15030,train,7dcf1fb8895f4ee61803d56f0ddff732,9d2dd41b0d471e1d
Normal-5882.png,Normal,Normal/images/Normal-5882.png,Normal/masks/Normal-5882.png,True,G15031,train,d103d4950ac4e06392825d927107f61a,99798e1e1a0f1e39
Normal-5883.png,Normal,Normal/images/Normal-5883.png,Normal/masks/Normal-5883.png,True,G15032,train,16d1e5c8bc47dd97b2d8b264d69a9ba3,996d861a5d5bb849
Normal-5884.png,Normal,Normal/images/Normal-5884.png,Normal/masks/Normal-5884.png,True,G15033,train,282b0a59e069a9030763d0d34ae60293,d97d520b1d446f19
Normal-5885.png,Normal,Normal/images/Normal-5885.png,Normal/masks/Normal-5885.png,True,G15034,train,e76e2d27d2deb85c9cad651fb9801289,9d0c500b5f693cbd
Normal-5886.png,Normal,Normal/images/Normal-5886.png,Normal/masks/Normal-5886.png,True,G15035,train,e228ca132c864ed2fe2603061b7a8f19,91215e1e175e78b9
Normal-5887.png,Normal,Normal/images/Normal-5887.png,Normal/masks/Normal-5887.png,True,G15036,train,45f20d57eea22a93e3634a586c1152c0,d5c168ca2e3b1b35
Normal-5888.png,Normal,Normal/images/Normal-5888.png,Normal/masks/Normal-5888.png,True,G15037,train,018333ed08d63811cd512d535f1b8934,d42fc30b2f09bf05
Normal-5889.png,Normal,Normal/images/Normal-5889.png,Normal/masks/Normal-5889.png,True,G15038,val,c498f9560356e96ae490c87079d0856d,dd2c46431b4cec7c
Normal-589.png,Normal,Normal/images/Normal-589.png,Normal/masks/Normal-589.png,True,G15039,val,c326c2698752071285fc7834c68df066,906ec67acd2d68c6
Normal-5890.png,Normal,Normal/images/Normal-5890.png,Normal/masks/Normal-5890.png,True,G15040,val,ef6a27a572b5a75457ececab80046422,dc2dc25fc20b2d0f
Normal-5891.png,Normal,Normal/images/Normal-5891.png,Normal/masks/Normal-5891.png,True,G15041,train,2be1fa85044ec7f448037eb6e6c245af,cd2cd34b1d444e5d
Normal-5892.png,Normal,Normal/images/Normal-5892.png,Normal/masks/Normal-5892.png,True,G15042,test,8b7e6361787d1967ddb81de9badf291c,dd2cc3431b4f4749
Normal-5893.png,Normal,Normal/images/Normal-5893.png,Normal/masks/Normal-5893.png,True,G15043,val,269eee3d3b0e5f8ce2a6948e561dd9ca,8979971a091f7ab1
Normal-5894.png,Normal,Normal/images/Normal-5894.png,Normal/masks/Normal-5894.png,True,G15044,train,345d547d2fbe031a0ed26a22dded2593,d13dc2c31b5c4f1a
Normal-5895.png,Normal,Normal/images/Normal-5895.png,Normal/masks/Normal-5895.png,True,G15045,train,a60d6c22c81e7a2cf3aae95211ba012e,d86987171e4d5c1e
Normal-5896.png,Normal,Normal/images/Normal-5896.png,Normal/masks/Normal-5896.png,True,G15046,test,171f1384c3643383580e192b2987fe76,d42ec35b0d4e2f4c
Normal-5897.png,Normal,Normal/images/Normal-5897.png,Normal/masks/Normal-5897.png,True,G15047,test,fca8682cc677c49be3999ed0d89c2a44,91790f46191bd46f
Normal-5898.png,Normal,Normal/images/Normal-5898.png,Normal/masks/Normal-5898.png,True,G15048,train,9b4200eacbd2f180de4d4d5ab4dc5933,dc3f421f08293dbc
Normal-5899.png,Normal,Normal/images/Normal-5899.png,Normal/masks/Normal-5899.png,True,G15049,train,e084b78f5acf0f5d636755aac3c12c23,cc3cc34b0c0e2fbd
Normal-59.png,Normal,Normal/images/Normal-59.png,Normal/masks/Normal-59.png,True,G15050,train,9ac8fc82a478c1cd2e7a2a030c238fe7,8168d65b4e5e5a4e
Normal-590.png,Normal,Normal/images/Normal-590.png,Normal/masks/Normal-590.png,True,G15051,train,f1716b875df6e239cbf1ad8c36206795,912bc25af85939d9
Normal-5900.png,Normal,Normal/images/Normal-5900.png,Normal/masks/Normal-5900.png,True,G15052,train,313e71b6f947ebc7ddd54aa628b49060,9979051e5e4a7ab8
Normal-5901.png,Normal,Normal/images/Normal-5901.png,Normal/masks/Normal-5901.png,True,G15053,train,02b158af35af5e212c5a75a9e1a450ae,817816c65a5de8eb
Normal-5902.png,Normal,Normal/images/Normal-5902.png,Normal/masks/Normal-5902.png,True,G15054,train,c51e4a4a288ae5a9f6a8a0355d5fb6b7,897d061a7f68a1e9
Normal-5903.png,Normal,Normal/images/Normal-5903.png,Normal/masks/Normal-5903.png,True,G15055,train,2c674d3186934dce017778de41654b11,d92ac2d71e5c4d1a
Normal-5904.png,Normal,Normal/images/Normal-5904.png,Normal/masks/Normal-5904.png,True,G15056,train,511a515dfdea855355fd3664cd77202e,cc6c865a19557e5c
Normal-5905.png,Normal,Normal/images/Normal-5905.png,Normal/masks/Normal-5905.png,True,G15057,train,555d6171dba28b750bf1f4c81abe33ce,dd3d421f080f3da5
Normal-5906.png,Normal,Normal/images/Normal-5906.png,Normal/masks/Normal-5906.png,True,G15058,train,63830447c915a665fc70ae5bd131f714,c135e7c32d38e92c
Normal-5907.png,Normal,Normal/images/Normal-5907.png,Normal/masks/Normal-5907.png,True,G15059,train,d1a1e655767a90bd16b616c1734bf05a,897a17181e1f29e7
Normal-5908.png,Normal,Normal/images/Normal-5908.png,Normal/masks/Normal-5908.png,True,G15060,train,fc8e86826f7b6494c113101d48446cdb,dd2ac30b9caaadb0
Normal-5909.png,Normal,Normal/images/Normal-5909.png,Normal/masks/Normal-5909.png,True,G15061,val,d8e0e052d610bbb29b1950f99e34a427,d939d7165c02fd18
Normal-591.png,Normal,Normal/images/Normal-591.png,Normal/masks/Normal-591.png,True,G15062,val,186348ac36640dcf400802dabbe5d176,d06ec759c5e078d2
Normal-5910.png,Normal,Normal/images/Normal-5910.png,Normal/masks/Normal-5910.png,True,G15063,test,ad6439a3ff6f26be454cd03281059cc7,9969961e58ce78b1
Normal-5911.png,Normal,Normal/images/Normal-5911.png,Normal/masks/Normal-5911.png,True,G15064,test,b531a2d158005b7a6ff620d6dd391d21,886887571b475f1d
Normal-5912.png,Normal,Normal/images/Normal-5912.png,Normal/masks/Normal-5912.png,True,G15065,test,904a822266d854a526799ec48bf6a433,d50b4a0d1cadb4f3
Normal-5913.png,Normal,Normal/images/Normal-5913.png,Normal/masks/Normal-5913.png,True,G15066,train,f2a173c78b7162a9de1dd65534b60059,d978c40e970b1b3d
Normal-5914.png,Normal,Normal/images/Normal-5914.png,Normal/masks/Normal-5914.png,True,G15067,train,8cb55cc9d082c8cfe7673eff16f0083b,817956431b1d6d3b
Normal-5915.png,Normal,Normal/images/Normal-5915.png,Normal/masks/Normal-5915.png,True,G15068,train,e4a58d9cfd2881f04b930894cbb7cf46,8839c51e15471f9f
Normal-5916.png,Normal,Normal/images/Normal-5916.png,Normal/masks/Normal-5916.png,True,G15069,train,3867737ef5e9c3215936292ed6406094,d92dc71e554a1d45
Normal-5917.png,Normal,Normal/images/Normal-5917.png,Normal/masks/Normal-5917.png,True,G15070,train,29e5ec99a29da12158f97846d2aa89f1,dd2dc04b1f554a5c
Normal-5918.png,Normal,Normal/images/Normal-5918.png,Normal/masks/Normal-5918.png,True,G15071,train,671ec275493d6e8ed9999fb529db0b05,d43dc25355c5471e
Normal-5919.png,Normal,Normal/images/Normal-5919.png,Normal/masks/Normal-5919.png,True,G15072,train,26d12343ab9b2ef91bdbf970ac146b83,807a47965b4f7ae0
Normal-592.png,Normal,Normal/images/Normal-592.png,Normal/masks/Normal-592.png,True,G15073,test,4448112f368db3a8ab6b1ee6f1568d11,957cc71acd0d6887
Normal-5920.png,Normal,Normal/images/Normal-5920.png,Normal/masks/Normal-5920.png,True,G15074,train,f4fe291ab5108560416661af4aeecc83,8179c7161a596b1f
Normal-5921.png,Normal,Normal/images/Normal-5921.png,Normal/masks/Normal-5921.png,True,G15075,val,d8fa27c30d1c8536752c22f987f54436,ccb9c21b83cf19e4
Normal-5922.png,Normal,Normal/images/Normal-5922.png,Normal/masks/Normal-5922.png,True,G15076,train,817f40d23ae21506917a62831290e6a0,817957561a574f19
Normal-5923.png,Normal,Normal/images/Normal-5923.png,Normal/masks/Normal-5923.png,True,G15077,val,f0f854a7a42bdf4278972a8c039813fc,817e061a1fce3ab9
Normal-5924.png,Normal,Normal/images/Normal-5924.png,Normal/masks/Normal-5924.png,True,G15078,val,d7da4e8b8d4e49e3347bdf2259715139,9969c6561a0baf78
Normal-5925.png,Normal,Normal/images/Normal-5925.png,Normal/masks/Normal-5925.png,True,G15079,train,d850a90576d0632c7e74814dda961c0b,cd3cc20b850bbfa6
Normal-5926.png,Normal,Normal/images/Normal-5926.png,Normal/masks/Normal-5926.png,True,G15080,train,8196d68854905bc0797432235bc22c1f,dd2ac34d444d5c4f
Normal-5927.png,Normal,Normal/images/Normal-5927.png,Normal/masks/Normal-5927.png,True,G15081,test,895cb3a21d8237d77da67e3190f01759,d93cc61919ec3cb1
Normal-5928.png,Normal,Normal/images/Normal-5928.png,Normal/masks/Normal-5928.png,True,G15082,train,688c5b42bdd22b20bd951549546a836c,dd3f820f544f1949
Normal-5929.png,Normal,Normal/images/Normal-5929.png,Normal/masks/Normal-5929.png,True,G15083,train,3268b8771159eb6b36515859f3156909,cda8d2df464950cd
Normal-593.png,Normal,Normal/images/Normal-593.png,Normal/masks/Normal-593.png,True,G15084,train,8851a619b05c223906e16d459a2d23bd,917c877a8d2d48cd
Normal-5930.png,Normal,Normal/images/Normal-5930.png,Normal/masks/Normal-5930.png,True,G15085,test,702349c30529eea37b97f9bb5a672a38,9568870313477f7a
Normal-5931.png,Normal,Normal/images/Normal-5931.png,Normal/masks/Normal-5931.png,True,G15086,train,45b6c4db66b02a0739e28c784141baed,dd2cd60b5e1c4d19
Normal-5932.png,Normal,Normal/images/Normal-5932.png,Normal/masks/Normal-5932.png,True,G15087,train,42e784fb33f001709ea0c1352efdbcbe,dc3dc2531d402f3d
Normal-5933.png,Normal,Normal/images/Normal-5933.png,Normal/masks/Normal-5933.png,True,G15088,train,085b0e9191feea95948ea27f914207e9,8c1a431f0d2dbda7
Normal-5934.png,Normal,Normal/images/Normal-5934.png,Normal/masks/Normal-5934.png,True,G15089,train,0adb1d502f1746efec22a62af0eac89c,992b0616965e9ece
Normal-5935.png,Normal,Normal/images/Normal-5935.png,Normal/masks/Normal-5935.png,True,G15090,train,522958c5aa8d7725eb66ba3148db047c,d97b06825a1a4d5f
Normal-5936.png,Normal,Normal/images/Normal-5936.png,Normal/masks/Normal-5936.png,True,G15091,train,ded7fd7bfdc81035b17393832a0cfd47,dc2cd34b4b4b0f46
Normal-5937.png,Normal,Normal/images/Normal-5937.png,Normal/masks/Normal-5937.png,True,G15092,train,cac48c5a22d7189bcf1e7c6a7016cc7e,d97e464b1d481d3c
Normal-5938.png,Normal,Normal/images/Normal-5938.png,Normal/masks/Normal-5938.png,True,G15093,train,04f59da76c14f52213720664c70d14f6,897d061e18497bf9
Normal-5939.png,Normal,Normal/images/Normal-5939.png,Normal/masks/Normal-5939.png,True,G15094,train,78508ec4204c38196f955edd1afb43a3,d53c03011b4ffe39
Normal-594.png,Normal,Normal/images/Normal-594.png,Normal/masks/Normal-594.png,True,G15095,train,a9277a55382b95797c7d1702e4cc8fe2,91fea72b8d2c290e
Normal-5940.png,Normal,Normal/images/Normal-5940.png,Normal/masks/Normal-5940.png,True,G15096,train,f731dab7ccd40a119c8ba09eb1049d1f,9568061a1dec7e79
Normal-5941.png,Normal,Normal/images/Normal-5941.png,Normal/masks/Normal-5941.png,True,G15097,train,464c1ac4a7382dce58aabc28365b3292,d97d855215c71a1e
Normal-5942.png,Normal,Normal/images/Normal-5942.png,Normal/masks/Normal-5942.png,True,G15098,train,7914659619dc520ff51bf551557f3686,d42d06d71b0d4f1e
Normal-5943.png,Normal,Normal/images/Normal-5943.png,Normal/masks/Normal-5943.png,True,G15099,train,ada3748067dc820b955f00a5662cecad,d52e430f4c4f5d4c
Normal-5944.png,Normal,Normal/images/Normal-5944.png,Normal/masks/Normal-5944.png,True,G15100,val,90d7dabdb7815ed93b325e8c9082c289,996fc61a455e1b45
Normal-5945.png,Normal,Normal/images/Normal-5945.png,Normal/masks/Normal-5945.png,True,G15101,val,e56b8da7eece7aa753084ef754589250,8160465e1a5f5f3b
Normal-5946.png,Normal,Normal/images/Normal-5946.png,Normal/masks/Normal-5946.png,True,G15102,val,3829cbf2a447ed21121e3019138a2faf,897d865a1f4e5a45
Normal-5947.png,Normal,Normal/images/Normal-5947.png,Normal/masks/Normal-5947.png,True,G15103,test,622fae73029a0750bb3f1aeb4b75632f,b94712194d39edb8
Normal-5948.png,Normal,Normal/images/Normal-5948.png,Normal/masks/Normal-5948.png,True,G15104,test,4e0cf558122619826dc08494992823eb,c42ec68f0c6c4e6f
Normal-5949.png,Normal,Normal/images/Normal-5949.png,Normal/masks/Normal-5949.png,True,G15105,test,a3ef3f6743b2147ca77b00bd00d902ba,997f45161a4d1d1b
Normal-595.png,Normal,Normal/images/Normal-595.png,Normal/masks/Normal-595.png,True,G15106,train,7d99af96607694ab67a54c327482bc68,d07ac738995938dc
Normal-5950.png,Normal,Normal/images/Normal-5950.png,Normal/masks/Normal-5950.png,True,G15107,train,26a499f90470e79abf65c5b3be51eff2,c87f420f530f5a0f
Normal-5951.png,Normal,Normal/images/Normal-5951.png,Normal/masks/Normal-5951.png,True,G15108,train,9b7b38152a5b55a14bd72640c8153350,983f061e104f5f4f
Normal-5952.png,Normal,Normal/images/Normal-5952.png,Normal/masks/Normal-5952.png,True,G15109,train,29e84522ad03ee8a82cd863564408d76,dd2f461b094f4a6c
Normal-5953.png,Normal,Normal/images/Normal-5953.png,Normal/masks/Normal-5953.png,True,G15110,train,274752723f4bfb12c8403ec69d27dc50,d82cc2531f4e5b1d
Normal-5954.png,Normal,Normal/images/Normal-5954.png,Normal/masks/Normal-5954.png,True,G15111,train,78a477795b674cb5a3b63b67344b967f,9179c61e1a8f5a39
Normal-5955.png,Normal,Normal/images/Normal-5955.png,Normal/masks/Normal-5955.png,True,G15112,train,ca7329d96d8cf76fd76fe78242d4345e,d939c61a1d493ab9
Normal-5956.png,Normal,Normal/images/Normal-5956.png,Normal/masks/Normal-5956.png,True,G15113,val,2c75943934c56bd42425721d2d260444,9979871a811f38ed
Normal-5957.png,Normal,Normal/images/Normal-5957.png,Normal/masks/Normal-5957.png,True,G15114,train,ff09040e61864f7579a407041581ca38,f969879610cd1f98
Normal-5958.png,Normal,Normal/images/Normal-5958.png,Normal/masks/Normal-5958.png,True,G15115,train,3fb4a0881a6ef4c52b8e3f314a3a9d4c,897d861e4e1f194d
Normal-5959.png,Normal,Normal/images/Normal-5959.png,Normal/masks/Normal-5959.png,True,G15116,train,0263e65d0f65399864bb77d1b6132c8f,9179061a1b9f7a39
Normal-596.png,Normal,Normal/images/Normal-596.png,Normal/masks/Normal-596.png,True,G15117,train,5a4a91712a4ebf0fc6894d14e6b3c00b,d12ee20b9f4b1d52
Normal-5960.png,Normal,Normal/images/Normal-5960.png,Normal/masks/Normal-5960.png,True,G15118,train,bc810ec62bc9d7954052832554dbd76b,993947161a974f3c
Normal-5961.png,Normal,Normal/images/Normal-5961.png,Normal/masks/Normal-5961.png,True,G15119,train,042870a44c737d90dee2fbecdc1c784f,dd2cc25b1d471b1c
Normal-5962.png,Normal,Normal/images/Normal-5962.png,Normal/masks/Normal-5962.png,True,G15120,val,bbaf6d7f0e163be39c79d024b7d7e572,897d861e145f594d
Normal-5963.png,Normal,Normal/images/Normal-5963.png,Normal/masks/Normal-5963.png,True,G15121,train,3560ac6827d2070c54826d7e09ccdfe8,cd39c61f584d59e0
Normal-5964.png,Normal,Normal/images/Normal-5964.png,Normal/masks/Normal-5964.png,True,G15122,val,b9ec457dfe7c191664a6c3fb19695c84,8178461e3f477b85
Normal-5965.png,Normal,Normal/images/Normal-5965.png,Normal/masks/Normal-5965.png,True,G15123,train,1fc7880fa3daf65218fed0f9484d76ed,d93fd25f10425b1c
Normal-5966.png,Normal,Normal/images/Normal-5966.png,Normal/masks/Normal-5966.png,True,G15124,test,74b7bac2b4babb381e50bd88f5d31db9,893dc44a5b4d4b5d
Normal-5967.png,Normal,Normal/images/Normal-5967.png,Normal/masks/Normal-5967.png,True,G15125,test,654f6f541e68b33b369ee098d678b3bb,d17bc616544a5b3a
Normal-5968.png,Normal,Normal/images/Normal-5968.png,Normal/masks/Normal-5968.png,True,G15126,test,e1ed381a66e7ce2463909523d99568dd,9d2ec21b494e1ee9
Normal-5969.png,Normal,Normal/images/Normal-5969.png,Normal/masks/Normal-5969.png,True,G15127,train,ee29599f38ce341543b34f4b4c16f067,997946871b1dcd49
Normal-597.png,Normal,Normal/images/Normal-597.png,Normal/masks/Normal-597.png,True,G15128,train,bea653dafc2ca225a5a38579bf9cbff6,d100f70afd0d793c
Normal-5970.png,Normal,Normal/images/Normal-5970.png,Normal/masks/Normal-5970.png,True,G15129,train,21ee01cbf01046e2a419ba27c503e0cc,d93d44431bc54e3b
Normal-5971.png,Normal,Normal/images/Normal-5971.png,Normal/masks/Normal-5971.png,True,G15130,train,7c840501822ab2fbab8baec9f739678a,9168161f5a5d7978
Normal-5972.png,Normal,Normal/images/Normal-5972.png,Normal/masks/Normal-5972.png,True,G15131,train,cbd08774bfc59a2b102778554b376582,dc3c430b2d2cec3d
Normal-5973.png,Normal,Normal/images/Normal-5973.png,Normal/masks/Normal-5973.png,True,G15132,train,1962207a5b2ef219eac0099631031ed6,d93dc61b1b045d39
Normal-5974.png,Normal,Normal/images/Normal-5974.png,Normal/masks/Normal-5974.png,True,G15133,train,ae20e21a55555d94c6bb059babd27296,987b85861e8e4b1f
Normal-5975.png,Normal,Normal/images/Normal-5975.png,Normal/masks/Normal-5975.png,True,G15134,train,decee1cf761f44791d9cfddce29eeb74,cc3d420b0eca2ffc
Normal-5976.png,Normal,Normal/images/Normal-5976.png,Normal/masks/Normal-5976.png,True,G15135,train,edfa2de8f289dd161b80b0fb750ef1d8,d92d861a1d4f1bc6
Normal-5977.png,Normal,Normal/images/Normal-5977.png,Normal/masks/Normal-5977.png,True,G15136,train,c2f3dfa250ec4c88891d381371217cdb,91794e14195fe939
Normal-5978.png,Normal,Normal/images/Normal-5978.png,Normal/masks/Normal-5978.png,True,G15137,train,3db213bc31d7bd4334a559fde971e61e,dc2c025b5d4e3cf8
Normal-5979.png,Normal,Normal/images/Normal-5979.png,Normal/masks/Normal-5979.png,True,G15138,train,f91d0a07f03af270e5db337ae9b0f7db,807805961f1f5e5f
Normal-598.png,Normal,Normal/images/Normal-598.png,Normal/masks/Normal-598.png,True,G15139,val,daa231beb279585614c1b7ec7c25cd5a,947fab1b8b8f0c84
Normal-5980.png,Normal,Normal/images/Normal-5980.png,Normal/masks/Normal-5980.png,True,G15140,train,fc3ed994e0c49a2b3933f7e7e049b9b1,c03d4b4b5c5a2f3c
Normal-5981.png,Normal,Normal/images/Normal-5981.png,Normal/masks/Normal-5981.png,True,G15141,val,94604ba36a5eda6cb1daa0aec7f74734,d97b061e190db939
Normal-5982.png,Normal,Normal/images/Normal-5982.png,Normal/masks/Normal-5982.png,True,G15142,val,c61c41314ec7591d345c248dd95e4cf2,8868479a1bc57f99
Normal-5983.png,Normal,Normal/images/Normal-5983.png,Normal/masks/Normal-5983.png,True,G15143,test,71c6e1507532661db2dcc5f117bb748b,9979071add1a38e5
Normal-5984.png,Normal,Normal/images/Normal-5984.png,Normal/masks/Normal-5984.png,True,G15144,train,66454cebe2a965c73db50e85cc80d079,d42cc2435b4eaf39
Normal-5985.png,Normal,Normal/images/Normal-5985.png,Normal/masks/Normal-5985.png,True,G15145,train,7dc930e39aa3e66743a84b5ece1a2329,8979871618cf5e1b
Normal-5986.png,Normal,Normal/images/Normal-5986.png,Normal/masks/Normal-5986.png,True,G15146,train,ec7735198bc41aecf91364e92159925c,9b201e1a5b7973d1
Normal-5987.png,Normal,Normal/images/Normal-5987.png,Normal/masks/Normal-5987.png,True,G15147,train,ec2f5a923668db5090d180d6a0fa41d9,9978c11f86c734c7
Normal-5988.png,Normal,Normal/images/Normal-5988.png,Normal/masks/Normal-5988.png,True,G15148,test,87443ca6d508e737964cd586d6ef17d8,9969c51652575a5b
Normal-5989.png,Normal,Normal/images/Normal-5989.png,Normal/masks/Normal-5989.png,True,G15149,train,ce0f7897e6e2080db38dd74c43bb401f,c13bc61d164f1f45
Normal-599.png,Normal,Normal/images/Normal-599.png,Normal/masks/Normal-599.png,True,G15150,val,c958533b5e18ec33e951b234538ceda0,987dc31e82075aaf
Normal-5990.png,Normal,Normal/images/Normal-5990.png,Normal/masks/Normal-5990.png,True,G15151,val,2d76a8c61fe2e4a545f19d501df57cbe,d879c59657c61ac1
Normal-5991.png,Normal,Normal/images/Normal-5991.png,Normal/masks/Normal-5991.png,True,G15152,val,2e933eb46042cbbc187c48deb925b247,996dc41a194f3ce5
Normal-5992.png,Normal,Normal/images/Normal-5992.png,Normal/masks/Normal-5992.png,True,G15153,test,8532e274833b3cd36e6f697a42a74e6a,892f061a5f6a7a78
Normal-5993.png,Normal,Normal/images/Normal-5993.png,Normal/masks/Normal-5993.png,True,G15154,train,205d3d26f212c535e64fb0e3b261af46,887dc61a4b0f1f4d
Normal-5994.png,Normal,Normal/images/Normal-5994.png,Normal/masks/Normal-5994.png,True,G15155,train,b98da6e050f6bd6b53d2f0ced0ae58e7,9978061a5b4d787b
Normal-5995.png,Normal,Normal/images/Normal-5995.png,Normal/masks/Normal-5995.png,True,G15156,train,ef8215fb2cb1d54f3dcaeec8d668892f,dd2cc7430a4cae3d
Normal-5996.png,Normal,Normal/images/Normal-5996.png,Normal/masks/Normal-5996.png,True,G15157,train,6e56faf99ca3bcacd9076320c5c803bf,dd28d21f5d0a1dc6
Normal-5997.png,Normal,Normal/images/Normal-5997.png,Normal/masks/Normal-5997.png,True,G15158,train,caefaaee052dfe3ea4a6a6c0af0e43d3,91691e1b7be869a1
Normal-5998.png,Normal,Normal/images/Normal-5998.png,Normal/masks/Normal-5998.png,True,G15159,train,09a3397018db795d4bd90f02a902c149,8d0e035b4d2e78ed
Normal-5999.png,Normal,Normal/images/Normal-5999.png,Normal/masks/Normal-5999.png,True,G15160,train,0343bbce6925b7496304aac10328b817,dd2cc21b402f39f5
Normal-6.png,Normal,Normal/images/Normal-6.png,Normal/masks/Normal-6.png,True,G15161,train,c1ccb40667c7d63a2119f65616dffe5c,cc7c524b555b4d46
Normal-60.png,Normal,Normal/images/Normal-60.png,Normal/masks/Normal-60.png,True,G15162,train,cde4dd7e5f48c72842dea260381d11d1,9979c7161a453e78
Normal-600.png,Normal,Normal/images/Normal-600.png,Normal/masks/Normal-600.png,True,G15163,test,4380f317a31066b1cd01ffc129e0e2f4,912bc23ad96939d3
Normal-6000.png,Normal,Normal/images/Normal-6000.png,Normal/masks/Normal-6000.png,True,G15164,test,b419f220eedc588fcd0ab853882abd34,ccbcc3431dd2471b
Normal-6001.png,Normal,Normal/images/Normal-6001.png,Normal/masks/Normal-6001.png,True,G15165,test,77712412da1ee40690d5231cc0dd4dcd,d47d03c30f2e4e4e
Normal-6002.png,Normal,Normal/images/Normal-6002.png,Normal/masks/Normal-6002.png,True,G15166,test,1076304abd61115f921c6e313e0100e5,996d065a5f7868e1
Normal-6003.png,Normal,Normal/images/Normal-6003.png,Normal/masks/Normal-6003.png,True,G15167,train,5fc6c2055379d86420b74345e2f78c6b,d929c41f18cd0ebd
Normal-6004.png,Normal,Normal/images/Normal-6004.png,Normal/masks/Normal-6004.png,True,G15168,train,79287ac839372174ca7df243449249b9,9968861a1f873abb
Normal-6005.png,Normal,Normal/images/Normal-6005.png,Normal/masks/Normal-6005.png,True,G15169,train,18283f8164c63099e20b1423747dda31,817b06065b794d5f
Normal-6006.png,Normal,Normal/images/Normal-6006.png,Normal/masks/Normal-6006.png,True,G15170,train,1c012b557970e92ed2036f836c005fdb,9d3cc61b99c4b8b8
Normal-6007.png,Normal,Normal/images/Normal-6007.png,Normal/masks/Normal-6007.png,True,G15171,train,424d5c6bbf877f17a6d1a203ce6bd441,dd3c02420b8bcf6f
Normal-6008.png,Normal,Normal/images/Normal-6008.png,Normal/masks/Normal-6008.png,True,G15172,train,dcd1903e875c00c4947703f63d537632,d1396a861e1ad92f
Normal-6009.png,Normal,Normal/images/Normal-6009.png,Normal/masks/Normal-6009.png,True,G15173,train,7240f6ebcd7efc9559dbe30faf34a501,d928065f1b4d6d3a
Normal-601.png,Normal,Normal/images/Normal-601.png,Normal/masks/Normal-601.png,True,G15174,train,7c57c0b00c83f912ddf8e5342f76cb92,802f875ac85e79e9
Normal-6010.png,Normal,Normal/images/Normal-6010.png,Normal/masks/Normal-6010.png,True,G15175,train,5d02e03fcd7cbe9e6e4e248c5161e630,d43d42431b4f4f0f
Normal-6011.png,Normal,Normal/images/Normal-6011.png,Normal/masks/Normal-6011.png,True,G15176,train,5b2faa3dc95273fad2f96b0232e11953,dd3e425f144f0b1c
Normal-6012.png,Normal,Normal/images/Normal-6012.png,Normal/masks/Normal-6012.png,True,G15177,train,d19d8d3e0bf6358274c601a8ef327969,d86d8413d3555e65
Normal-6013.png,Normal,Normal/images/Normal-6013.png,Normal/masks/Normal-6013.png,True,G15178,train,9630117326e7ef3bbbd300c7c04ede03,dd2852131b4f1f4b
Normal-6014.png,Normal,Normal/images/Normal-6014.png,Normal/masks/Normal-6014.png,True,G15179,test,a8f195f79e17faca4d627ad936e65e1f,bd7c431f4c4d1c49
Normal-6015.png,Normal,Normal/images/Normal-6015.png,Normal/masks/Normal-6015.png,True,G15180,train,69bea1e4672470786ceeccf692239f62,d93b86121e5e6b19
Normal-6016.png,Normal,Normal/images/Normal-6016.png,Normal/masks/Normal-6016.png,True,G15181,train,e1be09b2ee9ea7fcaa49503f398c9fac,9979851e5e0d1a5b
Normal-6017.png,Normal,Normal/images/Normal-6017.png,Normal/masks/Normal-6017.png,True,G15182,val,04b8efeaafa370fca0a5d5278d7fde71,9179c7161ac71b1b
Normal-6018.png,Normal,Normal/images/Normal-6018.png,Normal/masks/Normal-6018.png,True,G15183,val,c34cffed6d4b530f0e94076af94cd6fc,95e8071a1f1c3f1d
Normal-6019.png,Normal,Normal/images/Normal-6019.png,Normal/masks/Normal-6019.png,True,G15184,val,3bd8916969e6b03d0a5c5e895ba700b8,91690e16185f7e79
Normal-602.png,Normal,Normal/images/Normal-602.png,Normal/masks/Normal-602.png,True,G15185,test,315dd76e7ff101a6e8410afa40c46303,d454e21adb9b38c9
Normal-6020.png,Normal,Normal/images/Normal-6020.png,Normal/masks/Normal-6020.png,True,G15186,train,03a76295365826d270cf1cfd30eb1971,81681e165b7ee879
Normal-6021.png,Normal,Normal/images/Normal-6021.png,Normal/masks/Normal-6021.png,True,G15187,train,3f2aaad7a53fc3f034da5502f760cc25,dd3cc24b1c4e1d78
Normal-6022.png,Normal,Normal/images/Normal-6022.png,Normal/masks/Normal-6022.png,True,G15188,train,921d26b1a64eadfb427f5e5a8c95f738,c03b561c1a3f3b2d
Normal-6023.png,Normal,Normal/images/Normal-6023.png,Normal/masks/Normal-6023.png,True,G15189,train,d4fc2cf2eb8546ae15d64b2f48234d94,81e826963a876e9f
Normal-6024.png,Normal,Normal/images/Normal-6024.png,Normal/masks/Normal-6024.png,True,G15190,train,e14a06d873ac37a0c3f47ac04b719f6c,997f471a1a4d1969
Normal-6025.png,Normal,Normal/images/Normal-6025.png,Normal/masks/Normal-6025.png,True,G15191,train,f03cec14e1d219e4802684926d544001,997b061e1e167ad8
Normal-6026.png,Normal,Normal/images/Normal-6026.png,Normal/masks/Normal-6026.png,True,G15192,train,754ea5476ad68e7d1e422254d8d5f3bd,997b07121a4f7ab8
Normal-6027.png,Normal,Normal/images/Normal-6027.png,Normal/masks/Normal-6027.png,True,G15193,train,f0d219512237677b3d68f2609b5c0038,80685f977f8166a3
Normal-6028.png,Normal,Normal/images/Normal-6028.png,Normal/masks/Normal-6028.png,True,G15194,train,26feeeabd5bbabb5ee6158aaaee3de9c,d93b57861e185b1a
Normal-6029.png,Normal,Normal/images/Normal-6029.png,Normal/masks/Normal-6029.png,True,G15195,train,a8864fddf392aee938c3b3f781b8650c,817807165af9b8f9
Normal-603.png,Normal,Normal/images/Normal-603.png,Normal/masks/Normal-603.png,True,G15196,train,118b8b1bdfacb47e598b978d470ca81e,917a5e7859593943
Normal-6030.png,Normal,Normal/images/Normal-6030.png,Normal/masks/Normal-6030.png,True,G15197,val,2cb3142cee3f5aa320081e4b63551a59,f879c7521e495919
Normal-6031.png,Normal,Normal/images/Normal-6031.png,Normal/masks/Normal-6031.png,True,G15198,val,abe8c3417133e4beaa8c9764d5a919a0,dd62d30b430b4f0f
Normal-6032.png,Normal,Normal/images/Normal-6032.png,Normal/masks/Normal-6032.png,True,G15199,val,65cf8d657cf4a202a09b03c0ecb61a50,fd2cc61a0d431b4f
Normal-6033.png,Normal,Normal/images/Normal-6033.png,Normal/masks/Normal-6033.png,True,G15200,train,01112681508260dbc83de1c7098857be,817b47161a075f1f
Normal-6034.png,Normal,Normal/images/Normal-6034.png,Normal/masks/Normal-6034.png,True,G15201,test,10e531553b49c08c3a17e329d448cb15,9920161f1d5e7f19
Normal-6035.png,Normal,Normal/images/Normal-6035.png,Normal/masks/Normal-6035.png,True,G15202,test,4ac49fea107ad85df0a8be85bf46f3a1,897d575a055a5959
Normal-6036.png,Normal,Normal/images/Normal-6036.png,Normal/masks/Normal-6036.png,True,G15203,train,97b90a114a3ecf9f4ea343182e161fed,817806d67a79e867
Normal-6037.png,Normal,Normal/images/Normal-6037.png,Normal/masks/Normal-6037.png,True,G15204,test,73d2432ede5ee5ecff5740ab90c0e589,dd39c65a1f460a1e
Normal-6038.png,Normal,Normal/images/Normal-6038.png,Normal/masks/Normal-6038.png,True,G15205,train,317fe74d996f6ffa7711362bfb1e5780,dd28d31b5d4e1949
Normal-6039.png,Normal,Normal/images/Normal-6039.png,Normal/masks/Normal-6039.png,True,G15206,train,d20cce99e22e65515b14ad4adfff4f8d,987b8497168e5f1a
Normal-604.png,Normal,Normal/images/Normal-604.png,Normal/masks/Normal-604.png,True,G15207,train,92260015e80a838fdbf841ba7745263a,c56be25acd4d09c5
Normal-6040.png,Normal,Normal/images/Normal-6040.png,Normal/masks/Normal-6040.png,True,G15208,train,90a74c0cc3adf80aea40ba97bc8cf271,997e46421b4d1f3c
Normal-6041.png,Normal,Normal/images/Normal-6041.png,Normal/masks/Normal-6041.png,True,G15209,train,5dd2c9279a76ab1c3c142d99774ce0eb,8c6d871b491d1ec7
Normal-6042.png,Normal,Normal/images/Normal-6042.png,Normal/masks/Normal-6042.png,True,G15210,train,e0fb8006968d214ea4e377ff21ec8edf,dd3d87170b051f49
Normal-6043.png,Normal,Normal/images/Normal-6043.png,Normal/masks/Normal-6043.png,True,G15211,train,44a9a300911ad858af41d88eea2e4717,dcf4c34b52c13bc2
Normal-6044.png,Normal,Normal/images/Normal-6044.png,Normal/masks/Normal-6044.png,True,G15212,train,f78170f88bc7bdf6b2cb49cb897f414b,dd2dc61e0c4e1e4d
Normal-6045.png,Normal,Normal/images/Normal-6045.png,Normal/masks/Normal-6045.png,True,G15213,train,2ff410d944c64fe284427a370d9dbb98,d9314e5b19d8a879
Normal-6046.png,Normal,Normal/images/Normal-6046.png,Normal/masks/Normal-6046.png,True,G15214,train,d7c44bd1726b3415a5b0ac47d3731b21,dc3d424d1d1c793c
Normal-6047.png,Normal,Normal/images/Normal-6047.png,Normal/masks/Normal-6047.png,True,G15215,train,eaffe3b793b4028ef46b48e0697a3ccc,dc2d52530b4f4d1d
Normal-6048.png,Normal,Normal/images/Normal-6048.png,Normal/masks/Normal-6048.png,True,G15216,train,d7c394774f5777ab35b82c827d5ef515,b87905961a594f5e
Normal-6049.png,Normal,Normal/images/Normal-6049.png,Normal/masks/Normal-6049.png,True,G15217,train,35d40eb3f51c944bc959c2b9048cd18d,81bec71d124f1f45
Normal-605.png,Normal,Normal/images/Normal-605.png,Normal/masks/Normal-605.png,True,G15218,train,423158220199d4ae4f1331508b3b9098,d02de638ec693ce1
Normal-6050.png,Normal,Normal/images/Normal-6050.png,Normal/masks/Normal-6050.png,True,G15219,train,9d495867f794df91e38cc314e2cd1f0e,9968171e1a1e7b39
Normal-6051.png,Normal,Normal/images/Normal-6051.png,Normal/masks/Normal-6051.png,True,G15220,val,899764dbca37c9c2b1b0ea39ab1c9818,992f061a5d58595f
Normal-6052.png,Normal,Normal/images/Normal-6052.png,Normal/masks/Normal-6052.png,True,G15221,val,0fdaacc6dd0cc154123bbe9104481fd2,9979061b592df8b1
Normal-6053.png,Normal,Normal/images/Normal-6053.png,Normal/masks/Normal-6053.png,True,G15222,train,0330276e635e6d876de531178fa0af33,897ec69217475e19
Normal-6054.png,Normal,Normal/images/Normal-6054.png,Normal/masks/Normal-6054.png,True,G15223,val,ba4caeddd348c6b69c97b8eb6628a24c,887ec65a5c691cec
Normal-6055.png,Normal,Normal/images/Normal-6055.png,Normal/masks/Normal-6055.png,True,G15224,test,2f7ff4be53ad0b35a5fafe95dd73012a,d97dd65a1a4e1b04
Normal-6056.png,Normal,Normal/images/Normal-6056.png,Normal/masks/Normal-6056.png,True,G15225,test,001342b3c95d30c9e4aa88bc35ba32a3,d91f475b194f0c1a
Normal-6057.png,Normal,Normal/images/Normal-6057.png,Normal/masks/Normal-6057.png,True,G15226,test,0760d7c8d503379a524829b371922512,99691e1e99a9a9b1
Normal-6058.png,Normal,Normal/images/Normal-6058.png,Normal/masks/Normal-6058.png,True,G15227,train,5513c7ab3c4c08a8668464cd3a3ae5f7,d12f461b1e455f1c
Normal-6059.png,Normal,Normal/images/Normal-6059.png,Normal/masks/Normal-6059.png,True,G15228,train,827c960eceadde0fd7639166b39c5528,993dc65f184a6f18
Normal-606.png,Normal,Normal/images/Normal-606.png,Normal/masks/Normal-606.png,True,G15229,train,a8ba2b98a2b53554e5f999dcb19791f0,8178c75ab8697969
Normal-6060.png,Normal,Normal/images/Normal-6060.png,Normal/masks/Normal-6060.png,True,G15230,train,16a5d2f008a89b7f705a5dba2e1e7f80,9c7e475f1d052c1c
Normal-6061.png,Normal,Normal/images/Normal-6061.png,Normal/masks/Normal-6061.png,True,G15231,train,cdf4a3b9ed29de5f68aa5efbfb22bf39,897b861e188d5bb9
Normal-6062.png,Normal,Normal/images/Normal-6062.png,Normal/masks/Normal-6062.png,True,G15232,train,d48b6005951056d011cbdd2dcce043f4,d52f831d461f194d
Normal-6063.png,Normal,Normal/images/Normal-6063.png,Normal/masks/Normal-6063.png,True,G15233,train,7ac43bfa682f82acac162788966f56d0,c97946471b514f1d
Normal-6064.png,Normal,Normal/images/Normal-6064.png,Normal/masks/Normal-6064.png,True,G15234,train,657eaf47d061fa30ecc0998975b4f8e3,997f025b0d2a39b5
Normal-6065.png,Normal,Normal/images/Normal-6065.png,Normal/masks/Normal-6065.png,True,G15235,train,298f5999fa4e40b2c022dfccf5d30e92,d03d46190f4e3d5e
Normal-6066.png,Normal,Normal/images/Normal-6066.png,Normal/masks/Normal-6066.png,True,G15236,train,1fd875ef523cee3d717b0dbebdcf816b,fd7dc61a5d411849
Normal-6067.png,Normal,Normal/images/Normal-6067.png,Normal/masks/Normal-6067.png,True,G15237,train,094b7d8117e98469dc0148d27ac797eb,916986161f5efa38
Normal-6068.png,Normal,Normal/images/Normal-6068.png,Normal/masks/Normal-6068.png,True,G15238,val,3bf0fc9c74c5b393d7e5c9af602cb6d5,993d471e4d4b1969
Normal-6069.png,Normal,Normal/images/Normal-6069.png,Normal/masks/Normal-6069.png,True,G15239,train,ba5f2d4a05d308d423f0291f2d927c3f,f52ac31b5c0e1e4e
Normal-607.png,Normal,Normal/images/Normal-607.png,Normal/masks/Normal-607.png,True,G15240,train,9d200aa1c72760913c78bbc26f65d489,d0eac7b9c9e838b0
Normal-6070.png,Normal,Normal/images/Normal-6070.png,Normal/masks/Normal-6070.png,True,G15241,val,740ffd891ff353cf24142cc77076cd9b,dc3d424f1e4e6919
Normal-6071.png,Normal,Normal/images/Normal-6071.png,Normal/masks/Normal-6071.png,True,G15242,val,dec7b255c5129213f334bcab2a42d348,d829851e8e5f1e4d
Normal-6072.png,Normal,Normal/images/Normal-6072.png,Normal/masks/Normal-6072.png,True,G15243,test,074dfcd5c9c64fae857827e5eeff3008,997e861a955f1a0d
Normal-6073.png,Normal,Normal/images/Normal-6073.png,Normal/masks/Normal-6073.png,True,G15244,train,0ee3497dd12d0bbfad56071903bdf956,817a96181c5b795f
Normal-6074.png,Normal,Normal/images/Normal-6074.png,Normal/masks/Normal-6074.png,True,G15245,test,c69dfac41315bc81282550895bdf2013,997d841b094e596f
Normal-6075.png,Normal,Normal/images/Normal-6075.png,Normal/masks/Normal-6075.png,True,G15246,train,02f7f15847df6879bf295e65eb046c2e,cd2e825d1b48fcf0
Normal-6076.png,Normal,Normal/images/Normal-6076.png,Normal/masks/Normal-6076.png,True,G15247,test,8f3aab5a6367225f47348e1b425e96be,cc2e504b4b4f4e3e
Normal-6077.png,Normal,Normal/images/Normal-6077.png,Normal/masks/Normal-6077.png,True,G15248,train,94518ee99542b9874b53178c03c68db8,d02fc6071a1c6f3b
Normal-6078.png,Normal,Normal/images/Normal-6078.png,Normal/masks/Normal-6078.png,True,G15249,val,5ea399489c171ea84b3f888c470195d4,997fc618540d39a7
Normal-6079.png,Normal,Normal/images/Normal-6079.png,Normal/masks/Normal-6079.png,True,G15250,val,aca15e52f65e48d2be13babc90958399,993907871e155f5a
Normal-608.png,Normal,Normal/images/Normal-608.png,Normal/masks/Normal-608.png,True,G15251,train,609dfca3ae6625c4881fc012c9c6f3f2,d47e871b8d0d6c86
Normal-6080.png,Normal,Normal/images/Normal-6080.png,Normal/masks/Normal-6080.png,True,G15252,train,6b43087c3432a42f47c9d0787b07e3bd,9d6cc25b850b1f8d
Normal-6081.png,Normal,Normal/images/Normal-6081.png,Normal/masks/Normal-6081.png,True,G15253,train,209b72af6c3a63c462c77846e6275c68,dc2d430e2c2d3ce7
Normal-6082.png,Normal,Normal/images/Normal-6082.png,Normal/masks/Normal-6082.png,True,G15254,train,323e5b764e6d2e7940118eaffc6831b7,91afc619461f1c4f
Normal-6083.png,Normal,Normal/images/Normal-6083.png,Normal/masks/Normal-6083.png,True,G15255,train,9c1eeb60df6eeabe23137901cec45093,dd3bd2071cc60b39
Normal-6084.png,Normal,Normal/images/Normal-6084.png,Normal/masks/Normal-6084.png,True,G15256,train,c004b017650da65e1572fdaeafe50ef6,853c17c35b1e0f4b
Normal-6085.png,Normal,Normal/images/Normal-6085.png,Normal/masks/Normal-6085.png,True,G15257,train,f17f59e820499b0604701f0d34df1e93,897bc7561b054f19
Normal-6086.png,Normal,Normal/images/Normal-6086.png,Normal/masks/Normal-6086.png,True,G15258,train,b31195508036f31ba856a58bc795ebf4,817b865e195e5a39
Normal-6087.png,Normal,Normal/images/Normal-6087.png,Normal/masks/Normal-6087.png,True,G15259,train,3757eea8225673147d45cbf90744b80a,9968075a1f4f5879
Normal-6088.png,Normal,Normal/images/Normal-6088.png,Normal/masks/Normal-6088.png,True,G15260,train,5fdb91977caaed5eb62dce4ff8cd834f,dc2c0f6dc4c493c7
Normal-6089.png,Normal,Normal/images/Normal-6089.png,Normal/masks/Normal-6089.png,True,G15261,train,ba70002ec0075410bf5c76a76b733e56,c13b16860b1dbecb
Normal-609.png,Normal,Normal/images/Normal-609.png,Normal/masks/Normal-609.png,True,G15262,val,a6fd442e6a553f05333eb6c440788fa9,917bd65e50593913
Normal-6090.png,Normal,Normal/images/Normal-6090.png,Normal/masks/Normal-6090.png,True,G15263,test,9ab76f82d11ef6988a0a7d41cbda57f3,86731d707262e66b
Normal-6091.png,Normal,Normal/images/Normal-6091.png,Normal/masks/Normal-6091.png,True,G15264,test,49c545551751d501ae3b47ca2ce71a2b,8742d14d4e436e6f
Normal-6092.png,Normal,Normal/images/Normal-6092.png,Normal/masks/Normal-6092.png,True,G15265,test,fb66124ba8f4d281c6e9e9edf69d1f0b,987d851e1e4b1cf8
Normal-6093.png,Normal,Normal/images/Normal-6093.png,Normal/masks/Normal-6093.png,True,G15266,train,752ccb2afff0d2cc9f324fe3ef37855a,c83f464b194d4f1d
Normal-6094.png,Normal,Normal/images/Normal-6094.png,Normal/masks/Normal-6094.png,True,G15267,train,43df5de6bc22793fc5c54c4897e5feb6,dc3cd3430fc28b0f
Normal-6095.png,Normal,Normal/images/Normal-6095.png,Normal/masks/Normal-6095.png,True,G15268,train,f1a6bcc4546acefc4292aa4205041e20,9979871a1b851f39
Normal-6096.png,Normal,Normal/images/Normal-6096.png,Normal/masks/Normal-6096.png,True,G15269,train,2aa665537941fb75c76287c2d898ad03,9968871a13cf5b1b
Normal-6097.png,Normal,Normal/images/Normal-6097.png,Normal/masks/Normal-6097.png,True,G15270,train,5ef536c0d2a32af1c3cd07b528781892,997907561a954f3a
Normal-6098.png,Normal,Normal/images/Normal-6098.png,Normal/masks/Normal-6098.png,True,G15271,train,b3dd7c6d09b56f64056c8ab6264f71ca,d13b56461a5a4d1f
Normal-6099.png,Normal,Normal/images/Normal-6099.png,Normal/masks/Normal-6099.png,True,G15272,train,a914fd2b7d1f95cbedf4a4d36cd4c217,d979c51655465e51
Normal-61.png,Normal,Normal/images/Normal-61.png,Normal/masks/Normal-61.png,True,G15273,train,f3893bb6642a7bac7a2e516cddfa6868,897fc71a184f5a19
Normal-610.png,Normal,Normal/images/Normal-610.png,Normal/masks/Normal-610.png,True,G15274,train,eb29823a3099f35824db3556b594e11f,81bac61aec693d4d
Normal-6100.png,Normal,Normal/images/Normal-6100.png,Normal/masks/Normal-6100.png,True,G15275,train,42ef96c6bff09f27404de85da2ea45de,8178161a7bfd6989
Normal-6101.png,Normal,Normal/images/Normal-6101.png,Normal/masks/Normal-6101.png,True,G15276,train,080564b453e123e639b54262700cfa07,9969461a1d4b3be5
Normal-6102.png,Normal,Normal/images/Normal-6102.png,Normal/masks/Normal-6102.png,True,G15277,train,e8a785d26cdc91084477310c1a64d50d,8d3d82931f0d0f1f
Normal-6103.png,Normal,Normal/images/Normal-6103.png,Normal/masks/Normal-6103.png,True,G15278,train,cea0e9e46557ff17cd22ff6fa8ec14fd,81391616187bff78
Normal-6104.png,Normal,Normal/images/Normal-6104.png,Normal/masks/Normal-6104.png,True,G15279,train,8b2689cb8b19f4f4317f123557fcae29,d97f461a1d465919
Normal-6105.png,Normal,Normal/images/Normal-6105.png,Normal/masks/Normal-6105.png,True,G15280,val,fddc1fe3b10ecc27340f6794e81fa054,806f47084f29efc7
Normal-6106.png,Normal,Normal/images/Normal-6106.png,Normal/masks/Normal-6106.png,True,G15281,train,b4235e5ad47616a279162d94ac4283d0,887d86161b4d5f78
Normal-6107.png,Normal,Normal/images/Normal-6107.png,Normal/masks/Normal-6107.png,True,G15282,val,3d0ed7722fc66638af2453be66b853c1,8c4cc70f786ce671
Normal-6108.png,Normal,Normal/images/Normal-6108.png,Normal/masks/Normal-6108.png,True,G15283,test,fe130ef0ec1401114c0ad217fda56980,d92d871b19451f1d
Normal-6109.png,Normal,Normal/images/Normal-6109.png,Normal/masks/Normal-6109.png,True,G15284,val,3540f691599eb4fd2cb771ea56476d1d,997dc61a504e3a57
Normal-611.png,Normal,Normal/images/Normal-611.png,Normal/masks/Normal-611.png,True,G15285,test,8b1b9390d2539065138c3c90477cc28d,d074c61b9b8d39b4
Normal-6110.png,Normal,Normal/images/Normal-6110.png,Normal/masks/Normal-6110.png,True,G15286,test,f33dcf1cb9764f9d1c43f1b983a466d4,99acc61ac61f314f
Normal-6111.png,Normal,Normal/images/Normal-6111.png,Normal/masks/Normal-6111.png,True,G15287,train,c0dbf1d9b18d0ab83d572639a5dd512e,c86d868b1b4b4f1d
Normal-6112.png,Normal,Normal/images/Normal-6112.png,Normal/masks/Normal-6112.png,True,G15288,train,75e4189affee48ca4652999a5ae4f24a,9869861e845fdac7
Normal-6113.png,Normal,Normal/images/Normal-6113.png,Normal/masks/Normal-6113.png,True,G15289,train,76f8dcd95e292878985adcdd44aba7b2,897d865e5b04fe18
Normal-6114.png,Normal,Normal/images/Normal-6114.png,Normal/masks/Normal-6114.png,True,G15290,train,346f7cff80ca3ab37dc5bb4fe5d9914b,9178860a1b7d6e79
Normal-6115.png,Normal,Normal/images/Normal-6115.png,Normal/masks/Normal-6115.png,True,G15291,test,0b550b8478e57ee9b3a83e2e697b053b,dd2cd24b14431f3d
Normal-6116.png,Normal,Normal/images/Normal-6116.png,Normal/masks/Normal-6116.png,True,G15292,train,7fe6ab2f147d96324cc5951549d734e0,d928c61e0e4e595f
Normal-6117.png,Normal,Normal/images/Normal-6117.png,Normal/masks/Normal-6117.png,True,G15293,train,dedf88c0a6b46ea7ce8316ca06511a56,993b53161a5d3b38
Normal-6118.png,Normal,Normal/images/Normal-6118.png,Normal/masks/Normal-6118.png,True,G15294,train,2cfb88a9ceb5cb24237f55daf23331f1,9879c58694c7b5d4
Normal-6119.png,Normal,Normal/images/Normal-6119.png,Normal/masks/Normal-6119.png,True,G15295,train,f76667d4e77ed0e6972fa3b3a9c99cb5,d93984461b1d4f5b
Normal-612.png,Normal,Normal/images/Normal-612.png,Normal/masks/Normal-612.png,True,G15296,train,414ef0185b01b617ec6d086cfa4c2345,907faf3e0b0b384c
Normal-6120.png,Normal,Normal/images/Normal-6120.png,Normal/masks/Normal-6120.png,True,G15297,train,f36666cb52e474ba69578d9b2fa380b5,997886125f475e1b
Normal-6121.png,Normal,Normal/images/Normal-6121.png,Normal/masks/Normal-6121.png,True,G15298,train,bd043951e068ea09ce0d519ece0f6e2f,d17986ca1e4a0b7e
Normal-6122.png,Normal,Normal/images/Normal-6122.png,Normal/masks/Normal-6122.png,True,G15299,train,07aa00b8b5bcd4a5a2414837a0b77f79,997bc7865e480a1f
Normal-6123.png,Normal,Normal/images/Normal-6123.png,Normal/masks/Normal-6123.png,True,G15300,train,ca4f5b4c5a917527eccc8f60368b729e,9959ce1859495cd7
Normal-6124.png,Normal,Normal/images/Normal-6124.png,Normal/masks/Normal-6124.png,True,G15301,val,ef74e95e4fca905717c7023f2e91f99b,8879c71e194d7a78
Normal-6125.png,Normal,Normal/images/Normal-6125.png,Normal/masks/Normal-6125.png,True,G15302,train,70b6d3e1b7cfb5d91698c4c2a371818f,d93dc60696471f32
Normal-6126.png,Normal,Normal/images/Normal-6126.png,Normal/masks/Normal-6126.png,True,G15303,train,d96e0f7575e09bf385a7453f2c61220d,dc2e430b594f2e5c
Normal-6127.png,Normal,Normal/images/Normal-6127.png,Normal/masks/Normal-6127.png,True,G15304,val,d68f9e11da1fb6e6d6cf92795748139c,9958d3595d0a39ae
Normal-6128.png,Normal,Normal/images/Normal-6128.png,Normal/masks/Normal-6128.png,True,G15305,val,f6582a396e223925cda27d0841b318fd,d939060a191def79
Normal-6129.png,Normal,Normal/images/Normal-6129.png,Normal/masks/Normal-6129.png,True,G15306,train,485f0800f6db86fd519eeaf69238c993,917a07021a5f7ed9
Normal-613.png,Normal,Normal/images/Normal-613.png,Normal/masks/Normal-613.png,True,G15307,train,5f72a5b6783914d9b63c1aa9ea13340c,91d4c71ee98d6986
Normal-6130.png,Normal,Normal/images/Normal-6130.png,Normal/masks/Normal-6130.png,True,G15308,val,d3e507cec230b8dd2a179c6209871def,9d6c874319464e7b
Normal-6131.png,Normal,Normal/images/Normal-6131.png,Normal/masks/Normal-6131.png,True,G15309,train,469183f6727d541ca5f2f48e954a6c3b,997d57175e4d0918
Normal-6132.png,Normal,Normal/images/Normal-6132.png,Normal/masks/Normal-6132.png,True,G15310,val,e38ab6f20e7a0a39f66ea0bc627570a8,d16d461f180db9b9
Normal-6133.png,Normal,Normal/images/Normal-6133.png,Normal/masks/Normal-6133.png,True,G15311,test,ad5f60c80c6a1be5b4d6b3ab9c69994d,d97b17161e550b1c
Normal-6134.png,Normal,Normal/images/Normal-6134.png,Normal/masks/Normal-6134.png,True,G15312,test,6347233144ea55f3a5b90de702a89320,997807061ad75b7a
Normal-6135.png,Normal,Normal/images/Normal-6135.png,Normal/masks/Normal-6135.png,True,G15313,train,10dc1f39c70a5d0d0e3813d7801da339,dd2c420b1c6cbcfc
Normal-6136.png,Normal,Normal/images/Normal-6136.png,Normal/masks/Normal-6136.png,True,G15314,train,f647d9b671271cb9a46f343132ccc2f1,c42f424f4b4e4f0f
Normal-6137.png,Normal,Normal/images/Normal-6137.png,Normal/masks/Normal-6137.png,True,G15315,train,8c29355cfabee35fbce7d76574da69ba,9969861e174e7879
Normal-6138.png,Normal,Normal/images/Normal-6138.png,Normal/masks/Normal-6138.png,True,G15316,train,431d4494831c504b2f5f91be84020d55,d43c43430d8fa7ad
Normal-6139.png,Normal,Normal/images/Normal-6139.png,Normal/masks/Normal-6139.png,True,G15317,train,080f412c83b417ef7ce62783d7c9d4b2,817c8e55395de82b
Normal-614.png,Normal,Normal/images/Normal-614.png,Normal/masks/Normal-614.png,True,G15318,train,21e582ef95b4913402afc760f4283334,947e871aa9852dec
Normal-6140.png,Normal,Normal/images/Normal-6140.png,Normal/masks/Normal-6140.png,True,G15319,train,1d529bd3ba2ae46b3aaa9f8569230640,95698e1a1b0d5d79
Normal-6141.png,Normal,Normal/images/Normal-6141.png,Normal/masks/Normal-6141.png,True,G15320,train,091e58208b67e6a93c5f431687edf16c,937a161f1a3b3a38
Normal-6142.png,Normal,Normal/images/Normal-6142.png,Normal/masks/Normal-6142.png,True,G15321,train,b60c928502ef8c26fb36424fdbdad762,dd2c920b4d4e1f4d
Normal-6143.png,Normal,Normal/images/Normal-6143.png,Normal/masks/Normal-6143.png,True,G15322,train,9946562204d4fdbf58f470e8f0e0ead5,83701d3f70616767
Normal-6144.png,Normal,Normal/images/Normal-6144.png,Normal/masks/Normal-6144.png,True,G15323,val,8108997fd12f4ec73284e54645e56b9a,d96f461f584f1909
Normal-6145.png,Normal,Normal/images/Normal-6145.png,Normal/masks/Normal-6145.png,True,G15324,test,bc7d5016b364737c68c8bb38c6250d8b,9d6ec016524f1f5a
Normal-6146.png,Normal,Normal/images/Normal-6146.png,Normal/masks/Normal-6146.png,True,G15325,test,3724718112ed6e0338fbbb7391b60412,887c530b5da43f99
Normal-6147.png,Normal,Normal/images/Normal-6147.png,Normal/masks/Normal-6147.png,True,G15326,test,dcd4f9d03adba2222c0a2bca0f07ab98,cc3d8313150f1f9d
Normal-6148.png,Normal,Normal/images/Normal-6148.png,Normal/masks/Normal-6148.png,True,G15327,train,ae79bfbd5b15912a243148c2d3cd78f7,d16fc61a4e0a1b4f
Normal-6149.png,Normal,Normal/images/Normal-6149.png,Normal/masks/Normal-6149.png,True,G15328,train,e8173c02cca359ff1fd819a15cd7b325,956c06595dec78a5
Normal-615.png,Normal,Normal/images/Normal-615.png,Normal/masks/Normal-615.png,True,G15329,train,4f8b2157f9244b82eb1d28bf166f087d,903dc71e99dd3894
Normal-6150.png,Normal,Normal/images/Normal-6150.png,Normal/masks/Normal-6150.png,True,G15330,train,82cb6956742ab2d16afa415ba7a8e3d5,d13d82521f8e1b1f
Normal-6151.png,Normal,Normal/images/Normal-6151.png,Normal/masks/Normal-6151.png,True,G15331,train,394a5f19aa8f48fd5d879dbd3b164447,cd4c42534f4f6c2e
Normal-6152.png,Normal,Normal/images/Normal-6152.png,Normal/masks/Normal-6152.png,True,G15332,train,01ab298f581329330f835b8cb6d5c9f9,d928c25b4f4b1f49
Normal-6153.png,Normal,Normal/images/Normal-6153.png,Normal/masks/Normal-6153.png,True,G15333,val,2a68af1661e87ea0a6677cf86f3025d2,9978060a38ede9bd
Normal-6154.png,Normal,Normal/images/Normal-6154.png,Normal/masks/Normal-6154.png,True,G15334,train,2fcea865e83e5841ec81a4903b18807b,dd28c61ae9e0b8e5
Normal-6155.png,Normal,Normal/images/Normal-6155.png,Normal/masks/Normal-6155.png,True,G15335,train,0da13e11bc09e31b95ca6ecfaa537a17,dd2cc21f540a3b6d
Normal-6156.png,Normal,Normal/images/Normal-6156.png,Normal/masks/Normal-6156.png,True,G15336,test,d4b29907a371e336949849836c09ccca,d52f82d31d1c0f1d
Normal-6157.png,Normal,Normal/images/Normal-6157.png,Normal/masks/Normal-6157.png,True,G15337,train,30ea3840e4157cfc3babef6cf1fed3d3,dd2ec24b4d442e3d
Normal-6158.png,Normal,Normal/images/Normal-6158.png,Normal/masks/Normal-6158.png,True,G15338,train,82d57e65c70952cb4d9dd7c7530db72e,c93dc31b18493d79
Normal-6159.png,Normal,Normal/images/Normal-6159.png,Normal/masks/Normal-6159.png,True,G15339,train,09a29f69dcd071a01638458764c538e7,9968861a5d7a394f
Normal-616.png,Normal,Normal/images/Normal-616.png,Normal/masks/Normal-616.png,True,G15340,train,193c32eed0a66af00fffe67c687408aa,d47ca31fcb2d2c84
Normal-6160.png,Normal,Normal/images/Normal-6160.png,Normal/masks/Normal-6160.png,True,G15341,test,c7c7503549c143f88bb0079e6fb8addd,d97946c6da5a544a
Normal-6161.png,Normal,Normal/images/Normal-6161.png,Normal/masks/Normal-6161.png,True,G15342,train,aa6a5f05764119a9db5fdb6e028f6531,8c2ec30b0f6e6c6e
Normal-6162.png,Normal,Normal/images/Normal-6162.png,Normal/masks/Normal-6162.png,True,G15343,train,088d8f698d0b7fd3d0666a243492b6d8,dc2dc2430fcf0b1d
Normal-6163.png,Normal,Normal/images/Normal-6163.png,Normal/masks/Normal-6163.png,True,G15344,train,59759e6d98b95a4236bf052bdf0ad79a,996d861a491dd8cf
Normal-6164.png,Normal,Normal/images/Normal-6164.png,Normal/masks/Normal-6164.png,True,G15345,val,e86f579a996bfa5da951cdf9077c75f6,d97b561a1e0b398a
Normal-6165.png,Normal,Normal/images/Normal-6165.png,Normal/masks/Normal-6165.png,True,G15346,val,729d16e93f624d5b1462a6c0f5b7f30a,f97fc61e190a198a
Normal-6166.png,Normal,Normal/images/Normal-6166.png,Normal/masks/Normal-6166.png,True,G15347,test,cf6a2e01c5a01d2ac8b47436c61d57f2,dc0b570c0f4d4a7b
Normal-6167.png,Normal,Normal/images/Normal-6167.png,Normal/masks/Normal-6167.png,True,G15348,test,11c09b864679f9ab02620afa249ac2de,997d071a095b39e9
Normal-6168.png,Normal,Normal/images/Normal-6168.png,Normal/masks/Normal-6168.png,True,G15349,train,1ec4a54c03650e4c771d93895d70cd66,d92d471e454e1b55
Normal-6169.png,Normal,Normal/images/Normal-6169.png,Normal/masks/Normal-6169.png,True,G15350,train,01abba6603c7fb3d9cb18e7e46eb4ab6,9979c71681453ed5
Normal-617.png,Normal,Normal/images/Normal-617.png,Normal/masks/Normal-617.png,True,G15351,train,e7efb01b306e8f00dfcf6ca09f0e17e9,913fe61a9c491dcc
Normal-6170.png,Normal,Normal/images/Normal-6170.png,Normal/masks/Normal-6170.png,True,G15352,train,580f8dd28544e55adc94204ca3a698fb,897f061a4f1a4f3a
Normal-6171.png,Normal,Normal/images/Normal-6171.png,Normal/masks/Normal-6171.png,True,G15353,val,902824ee5b8a12c809ed32ae84dbbea2,d03d4e571a1b6d49
Normal-6172.png,Normal,Normal/images/Normal-6172.png,Normal/masks/Normal-6172.png,True,G15354,train,8f26e7807be8c100acda329007db8bca,dd3dc2521b91464f
Normal-6173.png,Normal,Normal/images/Normal-6173.png,Normal/masks/Normal-6173.png,True,G15355,train,e8bbce5a43ffd9509a3e2e39b443e75c,893c86521f4e5a7d
Normal-6174.png,Normal,Normal/images/Normal-6174.png,Normal/masks/Normal-6174.png,True,G15356,train,4d7b40308c5fe2e6bfa38722587c619d,dd2ed4525b454a0f
Normal-6175.png,Normal,Normal/images/Normal-6175.png,Normal/masks/Normal-6175.png,True,G15357,train,f369b3c6c84673997277c6b8b3380d76,d87d42196c3879ec
Normal-6176.png,Normal,Normal/images/Normal-6176.png,Normal/masks/Normal-6176.png,True,G15358,train,9e637f8be61872b229dd710f5e216571,817906da59a9efa1
Normal-6177.png,Normal,Normal/images/Normal-6177.png,Normal/masks/Normal-6177.png,True,G15359,train,08feb5ea64c1593aeec77f8f45f22c55,bd74530b4c4e5c4e
Normal-6178.png,Normal,Normal/images/Normal-6178.png,Normal/masks/Normal-6178.png,True,G15360,train,2b2064387c946f7bc0446c15d28ae06f,d92d420f1e6c6db8
Normal-6179.png,Normal,Normal/images/Normal-6179.png,Normal/masks/Normal-6179.png,True,G15361,train,f63e8169d55f30a8bd0044f3e05923dc,9979861a5d1f7a11
Normal-618.png,Normal,Normal/images/Normal-618.png,Normal/masks/Normal-618.png,True,G15362,val,2840de22883e718c01754261921083cf,d476a71fc92568e0
Normal-6180.png,Normal,Normal/images/Normal-6180.png,Normal/masks/Normal-6180.png,True,G15363,val,fdafa4bd7de05aeda6b5e051be8695e0,d939c61a9a4f3929
Normal-6181.png,Normal,Normal/images/Normal-6181.png,Normal/masks/Normal-6181.png,True,G15364,train,2d4f60314206af052117b16bf9e9c1b0,817b04981b3b1f7b
Normal-6182.png,Normal,Normal/images/Normal-6182.png,Normal/masks/Normal-6182.png,True,G15365,test,9bf0479ea4973ad0ada1e4d329fb0e5f,9128965f585b7966
Normal-6183.png,Normal,Normal/images/Normal-6183.png,Normal/masks/Normal-6183.png,True,G15366,test,1bcc62a604b1b438881458876da753fb,997c4619ac2db0e7
Normal-6184.png,Normal,Normal/images/Normal-6184.png,Normal/masks/Normal-6184.png,True,G15367,train,a884d0692ac0a7902f3583873fd8f14d,992d46571e455e5a
Normal-6185.png,Normal,Normal/images/Normal-6185.png,Normal/masks/Normal-6185.png,True,G15368,train,a7f2a1b663ad2986b2083d5a28fdfc6b,d4bdc38b49c26f90
Normal-6186.png,Normal,Normal/images/Normal-6186.png,Normal/masks/Normal-6186.png,True,G15369,train,3de54a904e65abb2725caddf67b2b232,8138065a7b6df8e9
Normal-6187.png,Normal,Normal/images/Normal-6187.png,Normal/masks/Normal-6187.png,True,G15370,train,913b9091e252e4e6d73d61d7aa9cb3ae,dc2d4619486eb8e7
Normal-6188.png,Normal,Normal/images/Normal-6188.png,Normal/masks/Normal-6188.png,True,G15371,train,d6ac4cef73ebc74c0a948b78d68a67f8,817f961a455b194f
Normal-6189.png,Normal,Normal/images/Normal-6189.png,Normal/masks/Normal-6189.png,True,G15372,train,5b4e9c537faaf40c776033b496021fd3,d52fc6075b096b61
Normal-619.png,Normal,Normal/images/Normal-619.png,Normal/masks/Normal-619.png,True,G15373,val,59476dead18b54e6e9a9e1c6edbebdba,957c873cad2c4ccc
Normal-6190.png,Normal,Normal/images/Normal-6190.png,Normal/masks/Normal-6190.png,True,G15374,train,4bd63056a49428e26b83f2ea0071636c,dc2cd24b52431f5e
Normal-6191.png,Normal,Normal/images/Normal-6191.png,Normal/masks/Normal-6191.png,True,G15375,train,3ae199317a71a91d4e7c9916da0aa10d,d03d4217134f6f39
Normal-6192.png,Normal,Normal/images/Normal-6192.png,Normal/masks/Normal-6192.png,True,G15376,train,3a0b190d64fc508d75c23b69449af899,d928c71fd40f3c49
Normal-6193.png,Normal,Normal/images/Normal-6193.png,Normal/masks/Normal-6193.png,True,G15377,val,38a89272f02073f9589b7999626cba05,dc2e430d063fd16e
Normal-6194.png,Normal,Normal/images/Normal-6194.png,Normal/masks/Normal-6194.png,True,G15378,test,c8f620452f2f51dfed14db1fd4e55cfd,cc34c71f1e465c1b
Normal-6195.png,Normal,Normal/images/Normal-6195.png,Normal/masks/Normal-6195.png,True,G15379,train,5c827aa5923efee2166468bca8fe85e0,d93bd25b15421b1e
Normal-6196.png,Normal,Normal/images/Normal-6196.png,Normal/masks/Normal-6196.png,True,G15380,train,a6690494f98a48d04951cb072cb27e86,d979d61a180e39a7
Normal-6197.png,Normal,Normal/images/Normal-6197.png,Normal/masks/Normal-6197.png,True,G15381,train,e1dac05ec5226852173020b667a85868,817b561c5a4b3d1d
Normal-6198.png,Normal,Normal/images/Normal-6198.png,Normal/masks/Normal-6198.png,True,G15382,train,8eeeee65a53fa82580a3203b84faeb7c,dd3953421b4d491f
Normal-6199.png,Normal,Normal/images/Normal-6199.png,Normal/masks/Normal-6199.png,True,G15383,train,1afa6948bbef012d84ed1fb7e39d9046,99e9961817387b5c
Normal-62.png,Normal,Normal/images/Normal-62.png,Normal/masks/Normal-62.png,True,G15384,train,4b663defe17b94aa3cbd085f65863ac2,cc6ed35b414b7c44
Normal-620.png,Normal,Normal/images/Normal-620.png,Normal/masks/Normal-620.png,True,G15385,val,017aa476c1580af812122dc96fdbd3d6,912dc61ab8c91ede
Normal-6200.png,Normal,Normal/images/Normal-6200.png,Normal/masks/Normal-6200.png,True,G15386,test,fdf1cf5da11d53a789f1fd20740aaea2,cc3dc34f5a062d3c
Normal-6201.png,Normal,Normal/images/Normal-6201.png,Normal/masks/Normal-6201.png,True,G15387,test,1254227f9bf65cff6a9bbcba50055721,d939d51b1e041b3d
Normal-6202.png,Normal,Normal/images/Normal-6202.png,Normal/masks/Normal-6202.png,True,G15388,test,819a848eb809763700a7b2ab00e338ec,dc0a430d6c3cecbe
Normal-6203.png,Normal,Normal/images/Normal-6203.png,Normal/masks/Normal-6203.png,True,G15389,train,4d5cc8820443a46eea53818f447e4a62,dd2d82439b13cf43
Normal-6204.png,Normal,Normal/images/Normal-6204.png,Normal/masks/Normal-6204.png,True,G15390,train,cb4744b66a6a5e6c567687b59bb6e9d4,d02d430b3dece539
Normal-6205.png,Normal,Normal/images/Normal-6205.png,Normal/masks/Normal-6205.png,True,G15391,train,5c1c682358e72e6b155c7a3bead22653,8078475e1f4d1fb8
Normal-6206.png,Normal,Normal/images/Normal-6206.png,Normal/masks/Normal-6206.png,True,G15392,train,fd2277d3e1b13f5014c28e364897c59a,dd2a830f4d0f4d4e
Normal-6207.png,Normal,Normal/images/Normal-6207.png,Normal/masks/Normal-6207.png,True,G15393,train,84c6c393b5bf4b320fdf826a1b37a62c,dc2c460b0fc72a3f
Normal-6208.png,Normal,Normal/images/Normal-6208.png,Normal/masks/Normal-6208.png,True,G15394,train,2cd3b6f7b7638dbecce810df87da5f28,897997181d5d384f
Normal-6209.png,Normal,Normal/images/Normal-6209.png,Normal/masks/Normal-6209.png,True,G15395,train,7c9f42c175d9c0d2c984e6959de5f9de,917806de5b69ec2c
Normal-621.png,Normal,Normal/images/Normal-621.png,Normal/masks/Normal-621.png,True,G15396,train,0ceb46458a93e66c6eb356716408ba60,913aca38d85b1ade
Normal-6210.png,Normal,Normal/images/Normal-6210.png,Normal/masks/Normal-6210.png,True,G15397,train,0d83d9da889e7ee23ff3a0e1c160d0ea,81297e1e79a564d5
Normal-6211.png,Normal,Normal/images/Normal-6211.png,Normal/masks/Normal-6211.png,True,G15398,train,7483ed78452c7fd52977f3b378bb5353,9d78c31b450f386d
Normal-6212.png,Normal,Normal/images/Normal-6212.png,Normal/masks/Normal-6212.png,True,G15399,train,4f372b1f2095c263524fb229483067d7,856c07021d4d7e7f
Normal-6213.png,Normal,Normal/images/Normal-6213.png,Normal/masks/Normal-6213.png,True,G15400,train,0f9cdef8e95d564e6687f1eb109c8a1a,dd2cc70b1d4d4d1c
Normal-6214.png,Normal,Normal/images/Normal-6214.png,Normal/masks/Normal-6214.png,True,G15401,train,3bda3d1b76b319005c04c4b50a6ab012,887e05861f5d4e5e
Normal-6215.png,Normal,Normal/images/Normal-6215.png,Normal/masks/Normal-6215.png,True,G15402,val,a9460fab82eed22456879a483164338c,d939c64a1f4c691d
Normal-6216.png,Normal,Normal/images/Normal-6216.png,Normal/masks/Normal-6216.png,True,G15403,val,6d03dac3e716300da1a21d0a3aaf36c7,c13ccb4b18474d3f
Normal-6217.png,Normal,Normal/images/Normal-6217.png,Normal/masks/Normal-6217.png,True,G15404,test,1c6092175fbeaf33541bc5b4f1a8556f,992d061a79ed39e1
Normal-6218.png,Normal,Normal/images/Normal-6218.png,Normal/masks/Normal-6218.png,True,G15405,val,4b337a1dbc8a4afad684cb4b3d48eb0b,807f871a1b2d3c7c
Normal-6219.png,Normal,Normal/images/Normal-6219.png,Normal/masks/Normal-6219.png,True,G15406,test,91ee93768194913504d1c3f10880b498,99619e4e191b8e79
Normal-622.png,Normal,Normal/images/Normal-622.png,Normal/masks/Normal-622.png,True,G15407,train,dcb32581988f05d5d710abeb1d42fdb9,d07cc73b8c3c38c6
Normal-6220.png,Normal,Normal/images/Normal-6220.png,Normal/masks/Normal-6220.png,True,G15408,train,fc5cc3d385134116d866c00b56301b32,d13dc6431b1f444f
Normal-6221.png,Normal,Normal/images/Normal-6221.png,Normal/masks/Normal-6221.png,True,G15409,train,bef1e8d37a21d0bb412b32f26062a851,cd6cd10b5d021f5d
Normal-6222.png,Normal,Normal/images/Normal-6222.png,Normal/masks/Normal-6222.png,True,G15410,train,f4af3fa7644c77f04b07fcdac8e7c884,c93dc65b1c462b39
Normal-6223.png,Normal,Normal/images/Normal-6223.png,Normal/masks/Normal-6223.png,True,G15411,val,736d275c1d1730cb410b0361cc7c5bec,9978871a154d5b4f
Normal-6224.png,Normal,Normal/images/Normal-6224.png,Normal/masks/Normal-6224.png,True,G15412,train,ef586cc4469114c10f5d0a7e203e5069,d93d461b1b0c2bbc
Normal-6225.png,Normal,Normal/images/Normal-6225.png,Normal/masks/Normal-6225.png,True,G15413,train,512cbc73c6426d2672a3fb79aba8b10b,9839d41b5d1a3be8
Normal-6226.png,Normal,Normal/images/Normal-6226.png,Normal/masks/Normal-6226.png,True,G15414,train,8ef7cb335a496185fd554f7ea43858e2,99e8561f0a195b1f
Normal-6227.png,Normal,Normal/images/Normal-6227.png,Normal/masks/Normal-6227.png,True,G15415,train,d5fcda7d18dcd7a26128aa989a963108,986dc51e824f3e59
Normal-6228.png,Normal,Normal/images/Normal-6228.png,Normal/masks/Normal-6228.png,True,G15416,train,38abf2425f333019fddaa0576bd0568f,897dd61e5c4a192b
Normal-6229.png,Normal,Normal/images/Normal-6229.png,Normal/masks/Normal-6229.png,True,G15417,val,f2e72176489126b7d659571bff0db6a4,917a161a5a5a1b7b
Normal-623.png,Normal,Normal/images/Normal-623.png,Normal/masks/Normal-623.png,True,G15418,train,e62c932db3f8a56dcad092f20e8167d7,9164c67899ec7dc4
Normal-6230.png,Normal,Normal/images/Normal-6230.png,Normal/masks/Normal-6230.png,True,G15419,train,1f7cd8971fede44061c75d51b193441e,997d460a1f0d6d3c
Normal-6231.png,Normal,Normal/images/Normal-6231.png,Normal/masks/Normal-6231.png,True,G15420,train,9c55a3612a5348632746bb88c5f46261,8139d61e18175f3b
Normal-6232.png,Normal,Normal/images/Normal-6232.png,Normal/masks/Normal-6232.png,True,G15421,val,5df4102312ad75b618bacb397f35e820,986d871a194f5d59
Normal-6233.png,Normal,Normal/images/Normal-6233.png,Normal/masks/Normal-6233.png,True,G15422,train,65c34da01b837e34b0fe75c34a7affdd,896807165e5f5e5a
Normal-6234.png,Normal,Normal/images/Normal-6234.png,Normal/masks/Normal-6234.png,True,G15423,test,b34aa000835bbddf874804e1cb8a3b35,997d421e544f1d4b
Normal-6235.png,Normal,Normal/images/Normal-6235.png,Normal/masks/Normal-6235.png,True,G15424,val,4c731893da54a0217cc2d5f1487d2ad5,d42d82d35355d54b
Normal-6236.png,Normal,Normal/images/Normal-6236.png,Normal/masks/Normal-6236.png,True,G15425,train,a663dd315630712c71ffe23e09c21123,c1bdd68a3cc23bd0
Normal-6237.png,Normal,Normal/images/Normal-6237.png,Normal/masks/Normal-6237.png,True,G15426,test,439ad4eb1bd9de40a1ed27ab7a43ef7a,cd3f06451e4c4f1b
Normal-6238.png,Normal,Normal/images/Normal-6238.png,Normal/masks/Normal-6238.png,True,G15427,test,3ab6c3b58e2a2fc81aaef4de9fc8d8f2,dc2f469f1b456e08
Normal-6239.png,Normal,Normal/images/Normal-6239.png,Normal/masks/Normal-6239.png,True,G15428,train,4bf6b6892fbe40359bda4f5201321229,9b2b8c1d9c5d3239
Normal-624.png,Normal,Normal/images/Normal-624.png,Normal/masks/Normal-624.png,True,G15429,train,e46dd56ba938eac607668df0ef9b80cb,d07ec7199d5c18c6
Normal-6240.png,Normal,Normal/images/Normal-6240.png,Normal/masks/Normal-6240.png,True,G15430,train,00ac2a1b326cd5421c4b6c962cf50725,dd2dd30b0d451b3c
Normal-6241.png,Normal,Normal/images/Normal-6241.png,Normal/masks/Normal-6241.png,True,G15431,train,737498bfde898f086728109cf553561b,d93d461d1c8a3bb8
Normal-6242.png,Normal,Normal/images/Normal-6242.png,Normal/masks/Normal-6242.png,True,G15432,train,af96f37d90c1493977652beb3794b6fc,dd6e825b4d4e5868
Normal-6243.png,Normal,Normal/images/Normal-6243.png,Normal/masks/Normal-6243.png,True,G15433,train,c6c863281e89b1a787945c1214c59563,d42c431b0b4e7f6a
Normal-6244.png,Normal,Normal/images/Normal-6244.png,Normal/masks/Normal-6244.png,True,G15434,train,74813971b76f66cea9f7f7468da31daf,dd2ec20b0d2c3bbc
Normal-6245.png,Normal,Normal/images/Normal-6245.png,Normal/masks/Normal-6245.png,True,G15435,train,1125cdb0aaa5ef9b26a00b096b9ac877,817d065e59ece8b8
Normal-6246.png,Normal,Normal/images/Normal-6246.png,Normal/masks/Normal-6246.png,True,G15436,train,92a10a938e04a969582a772638b97340,d42f02c3df0a4c6f
Normal-6247.png,Normal,Normal/images/Normal-6247.png,Normal/masks/Normal-6247.png,True,G15437,train,d1a685237bedebe81a97868869019699,e92d8612471a5f4f
Normal-6248.png,Normal,Normal/images/Normal-6248.png,Normal/masks/Normal-6248.png,True,G15438,train,9e6e88d176aedc58789b733df53b2e41,997d861b1bc51f18
Normal-6249.png,Normal,Normal/images/Normal-6249.png,Normal/masks/Normal-6249.png,True,G15439,train,e7cada4dc8f356f3bf27528de69c658d,887d425b4d4f1c4f
Normal-625.png,Normal,Normal/images/Normal-625.png,Normal/masks/Normal-625.png,True,G15440,train,f5efacff0ccb704a21c48788cbaaedb2,c06cc75bcde549c1
Normal-6250.png,Normal,Normal/images/Normal-6250.png,Normal/masks/Normal-6250.png,True,G15441,train,99d38ae4008e755ab01a384163b9bb69,987d061e1f4a7a6a
Normal-6251.png,Normal,Normal/images/Normal-6251.png,Normal/masks/Normal-6251.png,True,G15442,val,c2177c39e6e166d96a3720ebe60f5e8e,9d2e061f1d5c4c7c
Normal-6252.png,Normal,Normal/images/Normal-6252.png,Normal/masks/Normal-6252.png,True,G15443,val,7b4935d4b97365fb74c62550d3802c5b,dd2fd20b1c4a1f58
Normal-6253.png,Normal,Normal/images/Normal-6253.png,Normal/masks/Normal-6253.png,True,G15444,test,24a271e1bb76a71f4dac8db783137cd9,9969071e1a4d7e78
Normal-6254.png,Normal,Normal/images/Normal-6254.png,Normal/masks/Normal-6254.png,True,G15445,test,9c2b6a8eb1713abfc0f75a8eae8b4d66,d17942d71b164e0f
Normal-6255.png,Normal,Normal/images/Normal-6255.png,Normal/masks/Normal-6255.png,True,G15446,test,169d2c258993b6cd8ea741287865ab63,d13c4fc3191b447d
Normal-6256.png,Normal,Normal/images/Normal-6256.png,Normal/masks/Normal-6256.png,True,G15447,train,90b297b65f382ae1821da7743f6e9b61,996f861a8f1a0e5e
Normal-6257.png,Normal,Normal/images/Normal-6257.png,Normal/masks/Normal-6257.png,True,G15448,train,906a38e1f46255d1ee4a2e44d8aa4128,c939461f164e395b
Normal-6258.png,Normal,Normal/images/Normal-6258.png,Normal/masks/Normal-6258.png,True,G15449,train,e18d5c98619dc8dc9e087a4f1f258811,b959061e195f3867
Normal-6259.png,Normal,Normal/images/Normal-6259.png,Normal/masks/Normal-6259.png,True,G15450,train,493f1f51bb9a37a2fb3637e4da348263,9879851e1a4d5f5a
Normal-626.png,Normal,Normal/images/Normal-626.png,Normal/masks/Normal-626.png,True,G15451,train,864e08f3ea39a99e2d38fe84021cadad,917ee71ea98c3885
Normal-6260.png,Normal,Normal/images/Normal-6260.png,Normal/masks/Normal-6260.png,True,G15452,train,dbeb665a77d64467f6d8116462870b82,d8ed061e471a555e
Normal-6261.png,Normal,Normal/images/Normal-6261.png,Normal/masks/Normal-6261.png,True,G15453,train,25004185ab0d7fd3abdb4f4a855bdd2c,b929879612471f57
Normal-6262.png,Normal,Normal/images/Normal-6262.png,Normal/masks/Normal-6262.png,True,G15454,val,ae6397c4555b267b2447bcd615d4be2e,9979c7121f4f1a19
Normal-6263.png,Normal,Normal/images/Normal-6263.png,Normal/masks/Normal-6263.png,True,G15455,train,0caf76c0de6f943f4793bd22824919bc,99781618197f79e1
Normal-6264.png,Normal,Normal/images/Normal-6264.png,Normal/masks/Normal-6264.png,True,G15456,train,c0f7f9df790b50c617547f7a7418f592,9978061a1b5d79e9
Normal-6265.png,Normal,Normal/images/Normal-6265.png,Normal/masks/Normal-6265.png,True,G15457,train,0e6e7c1a966c4bd5a520bea2b3355ed7,cc2c41131b5d4f5f
Normal-6266.png,Normal,Normal/images/Normal-6266.png,Normal/masks/Normal-6266.png,True,G15458,train,e039aa725e836a1cd8ec4785b60d5d0d,997bc615965e384a
Normal-6267.png,Normal,Normal/images/Normal-6267.png,Normal/masks/Normal-6267.png,True,G15459,train,e13d4b5280b1996d300dddc93d5e152d,ce3eb7c10b0ae2e1
Normal-6268.png,Normal,Normal/images/Normal-6268.png,Normal/masks/Normal-6268.png,True,G15460,val,464e4d82c14b11dc07f2f7beba27d5d4,912966de3bc53ab0
Normal-6269.png,Normal,Normal/images/Normal-6269.png,Normal/masks/Normal-6269.png,True,G15461,test,f9d207c035ff6d5cb521df6d647a6de7,846ec74b1f4e0a4f
Normal-627.png,Normal,Normal/images/Normal-627.png,Normal/masks/Normal-627.png,True,G15462,train,198ab995b4c7d52c0b1671df50290a95,d16fc739854f8143
Normal-6270.png,Normal,Normal/images/Normal-6270.png,Normal/masks/Normal-6270.png,True,G15463,train,657bd29b212036cd61cfa9cf86139961,8979861e185d78f9
Normal-6271.png,Normal,Normal/images/Normal-6271.png,Normal/masks/Normal-6271.png,True,G15464,val,dfb75bcb435060ea50114d00c98ab2fb,d96f061a155a595b
Normal-6272.png,Normal,Normal/images/Normal-6272.png,Normal/masks/Normal-6272.png,True,G15465,train,0d40fc9a51dd2b8f488c8fef424f6c82,d9290616c70e9adf
Normal-6273.png,Normal,Normal/images/Normal-6273.png,Normal/masks/Normal-6273.png,True,G15466,test,900240d9d9f99ed5861cfc191864a570,803f0416195d7d7f
Normal-6274.png,Normal,Normal/images/Normal-6274.png,Normal/masks/Normal-6274.png,True,G15467,test,bc73b941c5aa88ef59d7b998a7ff256d,993dc61a1c5a1d4f
Normal-6275.png,Normal,Normal/images/Normal-6275.png,Normal/masks/Normal-6275.png,True,G15468,train,eec270bbdb6c9706ed7912240429cb90,807f170b29552f3b
Normal-6276.png,Normal,Normal/images/Normal-6276.png,Normal/masks/Normal-6276.png,True,G15469,train,f1fd9155503a45f79a322a49ae333247,dd2870d5d2525f43
Normal-6277.png,Normal,Normal/images/Normal-6277.png,Normal/masks/Normal-6277.png,True,G15470,train,1ea1ee1026220a28a8d6da83a5066686,dd2c421b1d4f38b5
Normal-6278.png,Normal,Normal/images/Normal-6278.png,Normal/masks/Normal-6278.png,True,G15471,train,a2711f02ee1eddb94a88c1a6282a7d6a,893cc61a184f3fb9
Normal-6279.png,Normal,Normal/images/Normal-6279.png,Normal/masks/Normal-6279.png,True,G15472,val,32896038978154372149012838d6f603,d17946c65a1ac74e
Normal-628.png,Normal,Normal/images/Normal-628.png,Normal/masks/Normal-628.png,True,G15473,train,2be14528ee7299a253cced847dab3d08,8129c75abe683979
Normal-6280.png,Normal,Normal/images/Normal-6280.png,Normal/masks/Normal-6280.png,True,G15474,train,d4b62f6b22559f96e92530101072bcbc,993d04971b0fce1b
Normal-6281.png,Normal,Normal/images/Normal-6281.png,Normal/masks/Normal-6281.png,True,G15475,train,ff2936574c5a751c83d1d18a774a1d32,d43dc30d1d4e6e38
Normal-6282.png,Normal,Normal/images/Normal-6282.png,Normal/masks/Normal-6282.png,True,G15476,train,e4078f11880ed2f751dfe07671e8e815,996ad65a415c5a5b
Normal-6283.png,Normal,Normal/images/Normal-6283.png,Normal/masks/Normal-6283.png,True,G15477,train,6d87c37d71c754fbbea7428a8c846cb6,937f84145a5b51ea
Normal-6284.png,Normal,Normal/images/Normal-6284.png,Normal/masks/Normal-6284.png,True,G15478,train,9cec3b8ceea4b888962e4ee17eab06a3,cc3d430c8a2fbde4
Normal-6285.png,Normal,Normal/images/Normal-6285.png,Normal/masks/Normal-6285.png,True,G15479,train,9218532f4c9ce12a8928e3240777b4e7,957e835b450f4b0d
Normal-6286.png,Normal,Normal/images/Normal-6286.png,Normal/masks/Normal-6286.png,True,G15480,test,a9551fd476f980a7f7990edaa49f1115,d87b05061e1e5f1b
Normal-6287.png,Normal,Normal/images/Normal-6287.png,Normal/masks/Normal-6287.png,True,G15481,train,14436798d44bff38e4053873095aaf2d,9829871e5c4f5b4d
Normal-6288.png,Normal,Normal/images/Normal-6288.png,Normal/masks/Normal-6288.png,True,G15482,train,caf9d5cbc96c7ece822d4110e57ca601,81795f861a5a59cd
Normal-6289.png,Normal,Normal/images/Normal-6289.png,Normal/masks/Normal-6289.png,True,G15483,train,59091c8cebca107d06d81e6458570362,d9310e1f19ee69b0
Normal-629.png,Normal,Normal/images/Normal-629.png,Normal/masks/Normal-629.png,True,G15484,train,babf1160907b57d0f3f2cf6b2e591365,9075c71aec6c3ce4
Normal-6290.png,Normal,Normal/images/Normal-6290.png,Normal/masks/Normal-6290.png,True,G15485,val,95769c6f1aa84a36afc5281f13e9e02e,992e96184c5b78ee
Normal-6291.png,Normal,Normal/images/Normal-6291.png,Normal/masks/Normal-6291.png,True,G15486,val,332f6af656ed42eaa2fba2a0607ad3ce,d92cc61e855a1e4f
Normal-6292.png,Normal,Normal/images/Normal-6292.png,Normal/masks/Normal-6292.png,True,G15487,test,fc2504e6a4375681fc15f469370503a5,897b161a19fe5a38
Normal-6293.png,Normal,Normal/images/Normal-6293.png,Normal/masks/Normal-6293.png,True,G15488,test,172d0dc8f4649dae1148369affa134f0,8969c716ce073e4b
Normal-6294.png,Normal,Normal/images/Normal-6294.png,Normal/masks/Normal-6294.png,True,G15489,train,03f16712920c428fe796bab94c341098,dd2cd20bd7401f55
Normal-6295.png,Normal,Normal/images/Normal-6295.png,Normal/masks/Normal-6295.png,True,G15490,train,7b6289a8722c4b82e2eee7d6cdc60dc6,8c0f035b4d5f4c7c
Normal-6296.png,Normal,Normal/images/Normal-6296.png,Normal/masks/Normal-6296.png,True,G15491,train,462d606d4afc2000e17c4e45cc53f9ab,d93ad64a0c69693d
Normal-6297.png,Normal,Normal/images/Normal-6297.png,Normal/masks/Normal-6297.png,True,G15492,train,5a840a0bd1ef839a08bc480023060c99,d939c2d7d7424f10
Normal-6298.png,Normal,Normal/images/Normal-6298.png,Normal/masks/Normal-6298.png,True,G15493,train,f27f18ca2e0831fb9938554b7ad6adc1,897d861a1d4e1bae
Normal-6299.png,Normal,Normal/images/Normal-6299.png,Normal/masks/Normal-6299.png,True,G15494,train,166104abdaf963222d14d3f29a43d89e,9c2dc3171d4d6c5c
Normal-63.png,Normal,Normal/images/Normal-63.png,Normal/masks/Normal-63.png,True,G15495,train,05b8f40ac0cb429a4d73e8ed764b62ce,8129ca38697979cf
Normal-630.png,Normal,Normal/images/Normal-630.png,Normal/masks/Normal-630.png,True,G15496,train,e99e89200180b17ce3da78a03cb6bb18,d42dc61e8d4d1cce
Normal-6300.png,Normal,Normal/images/Normal-6300.png,Normal/masks/Normal-6300.png,True,G15497,train,5fef274ae4f26248bd67bb897fd7edb1,9169861a15467bf7
Normal-6301.png,Normal,Normal/images/Normal-6301.png,Normal/masks/Normal-6301.png,True,G15498,val,5e70ecad9f29bbe14325945d6151db32,dc2d52d35b4f414a
Normal-6302.png,Normal,Normal/images/Normal-6302.png,Normal/masks/Normal-6302.png,True,G15499,test,79f58f39bdc20230b9e340768e7437b9,d42e53530f0f0e4f
Normal-6303.png,Normal,Normal/images/Normal-6303.png,Normal/masks/Normal-6303.png,True,G15500,test,8417b8ac4c9333e761aba78378b96fa0,dd3e421f114e2f38
Normal-6304.png,Normal,Normal/images/Normal-6304.png,Normal/masks/Normal-6304.png,True,G15501,train,212c67e15e9885f0609b5c0420dae355,9969861e1add9a78
Normal-6305.png,Normal,Normal/images/Normal-6305.png,Normal/masks/Normal-6305.png,True,G15502,train,276cbbd1dc51e8e5800e2b5ae9f55a87,913e06586d7cdec2
Normal-6306.png,Normal,Normal/images/Normal-6306.png,Normal/masks/Normal-6306.png,True,G15503,train,d147982504942ddebc1920ac34804859,9969851e8a5eb8e9
Normal-6307.png,Normal,Normal/images/Normal-6307.png,Normal/masks/Normal-6307.png,True,G15504,train,a4bf0ca6dbccfd00016f7cee8cf273c3,896e065a4f786f68
Normal-6308.png,Normal,Normal/images/Normal-6308.png,Normal/masks/Normal-6308.png,True,G15505,val,94a403182830ba10c42e3f45e6c3dd8e,d96f861a095c594f
Normal-6309.png,Normal,Normal/images/Normal-6309.png,Normal/masks/Normal-6309.png,True,G15506,val,1647b269facfbf158e888838c0702e06,9979049e55ce1bc3
Normal-631.png,Normal,Normal/images/Normal-631.png,Normal/masks/Normal-631.png,True,G15507,train,b4aff8a08abcd533f4b665440c0092f9,9168c73aa96939cd
Normal-6310.png,Normal,Normal/images/Normal-6310.png,Normal/masks/Normal-6310.png,True,G15508,test,b760280c42dd19711a49ee87255fa2cc,9168175a5f5b5a58
Normal-6311.png,Normal,Normal/images/Normal-6311.png,Normal/masks/Normal-6311.png,True,G15509,train,3d4154e6e0deb712eb63930e8df9e66f,987dc71e451e154e
Normal-6312.png,Normal,Normal/images/Normal-6312.png,Normal/masks/Normal-6312.png,True,G15510,train,17b2796f4a9f1959ddbb0d74b5bf4698,887f871e085f194f
Normal-6313.png,Normal,Normal/images/Normal-6313.png,Normal/masks/Normal-6313.png,True,G15511,train,0541356522139dbd9384d9713eed0401,c07f47825d4d5d1c
Normal-6314.png,Normal,Normal/images/Normal-6314.png,Normal/masks/Normal-6314.png,True,G15512,train,43e2042d7a5cab4cc70055e62a0378a9,897d86161f4c5b39
Normal-6315.png,Normal,Normal/images/Normal-6315.png,Normal/masks/Normal-6315.png,True,G15513,train,cf50679536458af8ead2f30217ed9d16,9169ced29dc74e50
Normal-6316.png,Normal,Normal/images/Normal-6316.png,Normal/masks/Normal-6316.png,True,G15514,train,9ab6f47fe4aeda7ed33e085f569bc1c9,8975060a1adf7c79
Normal-6317.png,Normal,Normal/images/Normal-6317.png,Normal/masks/Normal-6317.png,True,G15515,train,d756180a5b5a791dd9ddef8218a722f6,dd2c421fc44b1c6f
Normal-6318.png,Normal,Normal/images/Normal-6318.png,Normal/masks/Normal-6318.png,True,G15516,train,cd151b444a9001e7faa71b609fd5222f,997947161b457a59
Normal-6319.png,Normal,Normal/images/Normal-6319.png,Normal/masks/Normal-6319.png,True,G15517,train,033967b0cd2e01954467b848b48614ee,882f561f4f1a5a6a
Normal-632.png,Normal,Normal/images/Normal-632.png,Normal/masks/Normal-632.png,True,G15518,train,ce93c33fc98ee7ba7c6b20f033d90747,9878c31adb1d59b4
Normal-6320.png,Normal,Normal/images/Normal-6320.png,Normal/masks/Normal-6320.png,True,G15519,train,e4ed9cdcbd82a84282a0c366c53d13e8,996886121bcf7ab9
Normal-6321.png,Normal,Normal/images/Normal-6321.png,Normal/masks/Normal-6321.png,True,G15520,train,f5881e8403b1ec750d86ca2cbf2ab45a,81eb961a057a3b4f
Normal-6322.png,Normal,Normal/images/Normal-6322.png,Normal/masks/Normal-6322.png,True,G15521,train,a20c966b6cad9e1d7663478c46e497c6,8168161a5b5b5b3f
Normal-6323.png,Normal,Normal/images/Normal-6323.png,Normal/masks/Normal-6323.png,True,G15522,test,a8f74cad5fa67d387c85753c0d8dde79,dc2fc30b4d4e1c4e
Normal-6324.png,Normal,Normal/images/Normal-6324.png,Normal/masks/Normal-6324.png,True,G15523,val,e562bff3b649fec413eceddfc9d3830c,dd2dc2539d422f19
Normal-6325.png,Normal,Normal/images/Normal-6325.png,Normal/masks/Normal-6325.png,True,G15524,val,aa4a866eafbe9dd2d98cb04a6a83b01d,9939c69e18693f38
Normal-6326.png,Normal,Normal/images/Normal-6326.png,Normal/masks/Normal-6326.png,True,G15525,test,a1dfebf1e7447fc0a2f499b126fc5a59,993d061e1c4d5f3c
Normal-6327.png,Normal,Normal/images/Normal-6327.png,Normal/masks/Normal-6327.png,True,G15526,test,a7ae3f9e5d12b91706977f0a0bad5971,dd6ec20b892e3c6c
Normal-6328.png,Normal,Normal/images/Normal-6328.png,Normal/masks/Normal-6328.png,True,G15527,train,416539079f2c71c2a3ed60ab6b63c091,d53b820a193d6f39
Normal-6329.png,Normal,Normal/images/Normal-6329.png,Normal/masks/Normal-6329.png,True,G15528,train,dcf3adc22d34387c3d1819b9cc3f2659,dd2b50571f504b1e
Normal-633.png,Normal,Normal/images/Normal-633.png,Normal/masks/Normal-633.png,True,G15529,val,c416c2b027b167970e774bed69ba4c9a,912be63a5e493d51
Normal-6330.png,Normal,Normal/images/Normal-6330.png,Normal/masks/Normal-6330.png,True,G15530,train,9fad63f2dd3717d8d81e85e6ddba1748,d97b871e194d581c
Normal-6331.png,Normal,Normal/images/Normal-6331.png,Normal/masks/Normal-6331.png,True,G15531,train,82addafa3662ee93535f9aa182235dd8,dd2a421f0a4b3d7c
Normal-6332.png,Normal,Normal/images/Normal-6332.png,Normal/masks/Normal-6332.png,True,G15532,train,c7c1fc42cb19bf5d2f6102bbc00f85b4,907a471e1e977e18
Normal-6333.png,Normal,Normal/images/Normal-6333.png,Normal/masks/Normal-6333.png,True,G15533,train,bfad774aa09f6d9fd9b1327b41bf25f7,d92906165ad77a3a
Normal-6334.png,Normal,Normal/images/Normal-6334.png,Normal/masks/Normal-6334.png,True,G15534,train,65b9593b975cadf42e69c8f0f49652ba,89685618175e797b
Normal-6335.png,Normal,Normal/images/Normal-6335.png,Normal/masks/Normal-6335.png,True,G15535,train,dd8391f3b8cc70e8605b0d5bee8c8df0,d92b161e1e465b39
Normal-6336.png,Normal,Normal/images/Normal-6336.png,Normal/masks/Normal-6336.png,True,G15536,train,d54daccf6d52422c46b018878aeb1e83,dc2c424b0bc47f3d
Normal-6337.png,Normal,Normal/images/Normal-6337.png,Normal/masks/Normal-6337.png,True,G15537,val,95fe5bc446ac87a600eec0346998693c,cc3cc20f1e6c3ce6
Normal-6338.png,Normal,Normal/images/Normal-6338.png,Normal/masks/Normal-6338.png,True,G15538,train,cdc5e529893bf9c59c969b27a9f866de,91780e1b7d6135e6
Normal-6339.png,Normal,Normal/images/Normal-6339.png,Normal/masks/Normal-6339.png,True,G15539,train,18eb94be1f533cb978b9d3236195a082,917d46c61b1d4c6d
Normal-634.png,Normal,Normal/images/Normal-634.png,Normal/masks/Normal-634.png,True,G15540,train,791d2d8adb0e4f15a813260cd0a25ad4,d42de30fbc481cdc
Normal-6340.png,Normal,Normal/images/Normal-6340.png,Normal/masks/Normal-6340.png,True,G15541,val,6f7e6577534d62108b096d07c220227f,d53e86531b474e19
Normal-6341.png,Normal,Normal/images/Normal-6341.png,Normal/masks/Normal-6341.png,True,G15542,val,5e72710ea31ab502d2011d29cb4ac0e2,97797a9a30967292
Normal-6342.png,Normal,Normal/images/Normal-6342.png,Normal/masks/Normal-6342.png,True,G15543,train,6b1acfc65df5d3977c07bc589145c418,9979c75a11475a59
Normal-6343.png,Normal,Normal/images/Normal-6343.png,Normal/masks/Normal-6343.png,True,G15544,test,1fb1a12e6153e58d4887d734bde944cd,817b5716185b3b39
Normal-6344.png,Normal,Normal/images/Normal-6344.png,Normal/masks/Normal-6344.png,True,G15545,test,2e3a4f2927e3749bddade4adebdccb0e,dd2ec20f5d4a0d6a
Normal-6345.png,Normal,Normal/images/Normal-6345.png,Normal/masks/Normal-6345.png,True,G15546,train,5342af23b84cb5f7ba8bd3fc84965858,cc3c4b0b2cecf43c
Normal-6346.png,Normal,Normal/images/Normal-6346.png,Normal/masks/Normal-6346.png,True,G15547,test,09fae52fff8e5e7fd84a6557fe62f525,d969521f411f514f
Normal-6347.png,Normal,Normal/images/Normal-6347.png,Normal/masks/Normal-6347.png,True,G15548,test,2d7ea32fc37d8e35cecb39e539463c00,c435434f0e4f4d6d
Normal-6348.png,Normal,Normal/images/Normal-6348.png,Normal/masks/Normal-6348.png,True,G15549,train,5625e9e03109e4d776c4d65151cb005b,c969861e18af3b39
Normal-6349.png,Normal,Normal/images/Normal-6349.png,Normal/masks/Normal-6349.png,True,G15550,train,3811cbfeb1ede0990f3d1ae9f31b08bc,dd3dc20dd24f1949
Normal-635.png,Normal,Normal/images/Normal-635.png,Normal/masks/Normal-635.png,True,G15551,train,5d0ea6632c4aec869eae273f5808f216,d1bee68e38c51c29
Normal-6350.png,Normal,Normal/images/Normal-6350.png,Normal/masks/Normal-6350.png,True,G15552,train,db3e1d0473a148b03e2c678e45a065dc,bd38c5831bcd431b
Normal-6351.png,Normal,Normal/images/Normal-6351.png,Normal/masks/Normal-6351.png,True,G15553,train,50b965b47f2b438500e3ae4a69e707eb,897f860a1f4c7a59
Normal-6352.png,Normal,Normal/images/Normal-6352.png,Normal/masks/Normal-6352.png,True,G15554,train,6dcd7ccd152f6fde33726b363fc1ee12,dd2ed20f140b3de4
Normal-6353.png,Normal,Normal/images/Normal-6353.png,Normal/masks/Normal-6353.png,True,G15555,train,8a88659c2f4669536a085bb50662b366,dd2d0b194c3cf8f0
Normal-6354.png,Normal,Normal/images/Normal-6354.png,Normal/masks/Normal-6354.png,True,G15556,train,2e9a1c267ca76b8506d50a3e47a8352d,dc2dc2035f02ef0f
Normal-6355.png,Normal,Normal/images/Normal-6355.png,Normal/masks/Normal-6355.png,True,G15557,train,74d4f98c222702608f856f0b8ef67b6e,9979071a1d4c7ab9
Normal-6356.png,Normal,Normal/images/Normal-6356.png,Normal/masks/Normal-6356.png,True,G15558,train,bef70024cc0b4871fbf0cc4866a78cef,dc2c470f0c4dec5d
Normal-6357.png,Normal,Normal/images/Normal-6357.png,Normal/masks/Normal-6357.png,True,G15559,train,f341fe5c9e6da446ce5e5254cdcf0160,d9a4471a475e194f
Normal-6358.png,Normal,Normal/images/Normal-6358.png,Normal/masks/Normal-6358.png,True,G15560,train,c74042c198979b907a3547c0be53f1dd,9978961b135f5a19
Normal-6359.png,Normal,Normal/images/Normal-6359.png,Normal/masks/Normal-6359.png,True,G15561,train,546ab94a813164b4795a3744fd879826,dd3ec20f182c3db4
Normal-636.png,Normal,Normal/images/Normal-636.png,Normal/masks/Normal-636.png,True,G15562,train,10686d5a5864b8e5f3a9451a6b208eee,945ecf9bcc9238c2
Normal-6360.png,Normal,Normal/images/Normal-6360.png,Normal/masks/Normal-6360.png,True,G15563,val,c6cd1065af11cd82af1d3848d7708a9e,d928061e1bc77e99
Normal-6361.png,Normal,Normal/images/Normal-6361.png,Normal/masks/Normal-6361.png,True,G15564,val,062f42aa91cd48ffba88c1eb7e5d84e0,dc2ec35b0dc60b1d
Normal-6362.png,Normal,Normal/images/Normal-6362.png,Normal/masks/Normal-6362.png,True,G15565,test,3d43534497cd97db457c5ed280d6710d,917b861a597e7a21
Normal-6363.png,Normal,Normal/images/Normal-6363.png,Normal/masks/Normal-6363.png,True,G15566,test,ac434e0bd9dbbfc8e2c7a0e55236e1cc,897b06161e9deab8
Normal-6364.png,Normal,Normal/images/Normal-6364.png,Normal/masks/Normal-6364.png,True,G15567,test,a172a54d46a85f62ca3cc842953455a5,8169de5e585f4938
Normal-6365.png,Normal,Normal/images/Normal-6365.png,Normal/masks/Normal-6365.png,True,G15568,val,622ee84851a647612682ee180604d11d,9968871a055e5b5f
Normal-6366.png,Normal,Normal/images/Normal-6366.png,Normal/masks/Normal-6366.png,True,G15569,train,ad496ce9163fa16dfab0c91be813b9c1,cc3cc30d0eed3cd4
Normal-6367.png,Normal,Normal/images/Normal-6367.png,Normal/masks/Normal-6367.png,True,G15570,train,49ef161019bcadcd5412e21e58461955,dc2cc34b4d4e2e3c
Normal-6368.png,Normal,Normal/images/Normal-6368.png,Normal/masks/Normal-6368.png,True,G15571,train,8a9444c55f9743297c937886782ddd38,8928065a5d7a7f78
Normal-6369.png,Normal,Normal/images/Normal-6369.png,Normal/masks/Normal-6369.png,True,G15572,val,e6e48c7426a25bfdb37aad5c46f07a5d,8179d61a174e394f
Normal-637.png,Normal,Normal/images/Normal-637.png,Normal/masks/Normal-637.png,True,G15573,val,5c408a888aa9619a97eb271f72b52b57,d1eec74aa94d29b0
Normal-6370.png,Normal,Normal/images/Normal-6370.png,Normal/masks/Normal-6370.png,True,G15574,val,ea21d44217d2bbce2953b7db8278477a,8168c656585b4f7b
Normal-6371.png,Normal,Normal/images/Normal-6371.png,Normal/masks/Normal-6371.png,True,G15575,test,ac616ec14268ed9f529a7d1a3010e05a,892e571fc44f184f
Normal-6372.png,Normal,Normal/images/Normal-6372.png,Normal/masks/Normal-6372.png,True,G15576,train,fbff87a1ac47833f27ac3c5d742990bb,9978861a1b9e6eb8
Normal-6373.png,Normal,Normal/images/Normal-6373.png,Normal/masks/Normal-6373.png,True,G15577,train,8932881a401b37806070d7cf5fcdeac2,992e9619c61b956e
Normal-6374.png,Normal,Normal/images/Normal-6374.png,Normal/masks/Normal-6374.png,True,G15578,train,bfb200f9b7cb818e9e2b0909bd3e4665,d5382a9730c4cf79
Normal-6375.png,Normal,Normal/images/Normal-6375.png,Normal/masks/Normal-6375.png,True,G15579,train,f4c287504fef19ad6b089ffa86453ccf,d07f560a5e48097f
Normal-6376.png,Normal,Normal/images/Normal-6376.png,Normal/masks/Normal-6376.png,True,G15580,train,83aad0828efbe17181947131f143ecb5,c87b97861ed45a19
Normal-6377.png,Normal,Normal/images/Normal-6377.png,Normal/masks/Normal-6377.png,True,G15581,train,3933135934483dc678db0b0a70c83ece,dc3e43098a2fb4ad
Normal-6378.png,Normal,Normal/images/Normal-6378.png,Normal/masks/Normal-6378.png,True,G15582,train,0f7fa3a94184c18e6ad39947066ea678,dd3cd24f544b3d08
Normal-6379.png,Normal,Normal/images/Normal-6379.png,Normal/masks/Normal-6379.png,True,G15583,train,230abf1af3b61427c0746889af668b24,d43d0a2f7c636272
Normal-638.png,Normal,Normal/images/Normal-638.png,Normal/masks/Normal-638.png,True,G15584,train,82373ba8095447c96ff13b7a3d798e11,91fd873a38693968
Normal-6380.png,Normal,Normal/images/Normal-6380.png,Normal/masks/Normal-6380.png,True,G15585,train,c62bb1e890d4ccb17e5b1beae581283a,817b171e181beb39
Normal-6381.png,Normal,Normal/images/Normal-6381.png,Normal/masks/Normal-6381.png,True,G15586,train,6137e8746b36dee5b9b7b13918da2030,997d44421b5d8f1b
Normal-6382.png,Normal,Normal/images/Normal-6382.png,Normal/masks/Normal-6382.png,True,G15587,test,08c24bbea56fa53eb9aeb973435d0575,dd7d52531f444a1c
Normal-6383.png,Normal,Normal/images/Normal-6383.png,Normal/masks/Normal-6383.png,True,G15588,train,1cac1097193f1e0c51ab2c2ad4fb3b41,9978c61b155a394f
Normal-6384.png,Normal,Normal/images/Normal-6384.png,Normal/masks/Normal-6384.png,True,G15589,train,f1e889e2684411c78ed28c0f8a6518df,dc2e821b45555b4f
Normal-6385.png,Normal,Normal/images/Normal-6385.png,Normal/masks/Normal-6385.png,True,G15590,train,2a0e40ebd37dcde66b4bc977a0f2dfbc,dd28d21f5a861f58
Normal-6386.png,Normal,Normal/images/Normal-6386.png,Normal/masks/Normal-6386.png,True,G15591,train,fa684ef3eb7028b0d4beeb512a2e4da7,9d2dc21bc50f194f
Normal-6387.png,Normal,Normal/images/Normal-6387.png,Normal/masks/Normal-6387.png,True,G15592,train,7cbe65313956fb23e12df30df438d128,996f161a5f28a8a7
Normal-6388.png,Normal,Normal/images/Normal-6388.png,Normal/masks/Normal-6388.png,True,G15593,train,119205c4662c01b395168d785dde77b8,dd2e40c31f556e49
Normal-6389.png,Normal,Normal/images/Normal-6389.png,Normal/masks/Normal-6389.png,True,G15594,train,52f2e60f588d9bc4e814cf2242c90d92,8178061a5defe8b5
Normal-639.png,Normal,Normal/images/Normal-639.png,Normal/masks/Normal-639.png,True,G15595,train,29cf460100fc02ac08d8e060fe2bd487,917cc67ae9a568e0
Normal-6390.png,Normal,Normal/images/Normal-6390.png,Normal/masks/Normal-6390.png,True,G15596,train,d5d14db5acc2d1d6076cc675af12a997,cc3d424f1c466dbc
Normal-6391.png,Normal,Normal/images/Normal-6391.png,Normal/masks/Normal-6391.png,True,G15597,val,3c8391a8496c420902bea84eaa895d0d,dc3d421b1b8a0fb9
Normal-6392.png,Normal,Normal/images/Normal-6392.png,Normal/masks/Normal-6392.png,True,G15598,train,d5c5960450160b62d20c26169d31e3e3,817b065e1a5d6e3a
Normal-6393.png,Normal,Normal/images/Normal-6393.png,Normal/masks/Normal-6393.png,True,G15599,train,5a5cc784cfcf794e65801c98811b0462,d528065a1f465f5b
Normal-6394.png,Normal,Normal/images/Normal-6394.png,Normal/masks/Normal-6394.png,True,G15600,train,deb9c1d62e0b1e5d7ca0daabe3b81d0b,9968861b1f59396d
Normal-6395.png,Normal,Normal/images/Normal-6395.png,Normal/masks/Normal-6395.png,True,G15601,train,02961fc293a8f32da790a4e64164bbdf,c139ce1e19465e79
Normal-6396.png,Normal,Normal/images/Normal-6396.png,Normal/masks/Normal-6396.png,True,G15602,train,1639c909dab528cbdc8d56e1773f0b7c,bc50d31b584f3a78
Normal-6397.png,Normal,Normal/images/Normal-6397.png,Normal/masks/Normal-6397.png,True,G15603,val,5ba9959b347c3dceef26ff26c57b21a6,993b841e5b495a79
Normal-6398.png,Normal,Normal/images/Normal-6398.png,Normal/masks/Normal-6398.png,True,G15604,val,a05ad941815e4bf7dfb835fa22190e16,dd2c424b574a1d6e
Normal-6399.png,Normal,Normal/images/Normal-6399.png,Normal/masks/Normal-6399.png,True,G15605,val,a46157449a318c710e45538b82fa5ccd,913cc6433a4fe91e
Normal-64.png,Normal,Normal/images/Normal-64.png,Normal/masks/Normal-64.png,True,G15606,test,92928067430258dad1b6d5c449c54fa3,c06cc73ecc4f2dc1
Normal-640.png,Normal,Normal/images/Normal-640.png,Normal/masks/Normal-640.png,True,G15607,test,cf76f0ce26f639651e2107b546d4617a,917cc61a894d69d7
Normal-6400.png,Normal,Normal/images/Normal-6400.png,Normal/masks/Normal-6400.png,True,G15608,train,bbabd9ead7fec04d9aabbe2f10fef584,8c4d431e4f5d1b47
Normal-6401.png,Normal,Normal/images/Normal-6401.png,Normal/masks/Normal-6401.png,True,G15609,test,a68bb5e4332836898ed9c3ced7dc2f96,f92fd6184619584f
Normal-6402.png,Normal,Normal/images/Normal-6402.png,Normal/masks/Normal-6402.png,True,G15610,train,a2b87b8b91baf182c6a3f4c211a5961a,8978465a1f4f196d
Normal-6403.png,Normal,Normal/images/Normal-6403.png,Normal/masks/Normal-6403.png,True,G15611,train,cc424f06eca12e41e713455b2adebdb6,d52c061b5d4e7c6a
Normal-6404.png,Normal,Normal/images/Normal-6404.png,Normal/masks/Normal-6404.png,True,G15612,train,165529d082de0d66506f6e189e19cda9,894f161a5c5a4efa
Normal-6405.png,Normal,Normal/images/Normal-6405.png,Normal/masks/Normal-6405.png,True,G15613,train,e7cde0d8a73e6c560856bcc9effa5d25,c13dd64f11464f1e
Normal-6406.png,Normal,Normal/images/Normal-6406.png,Normal/masks/Normal-6406.png,True,G15614,train,131c01497e15b7c8d1ba599e16a514a5,913b47165e57584d
Normal-6407.png,Normal,Normal/images/Normal-6407.png,Normal/masks/Normal-6407.png,True,G15615,train,cbdfd195dcba3ceb1c48171728795700,8179975a181f7a39
Normal-6408.png,Normal,Normal/images/Normal-6408.png,Normal/masks/Normal-6408.png,True,G15616,train,00076eb127acb0bdd4d989e177588c1e,c939c61f1c4a3d78
Normal-6409.png,Normal,Normal/images/Normal-6409.png,Normal/masks/Normal-6409.png,True,G15617,train,33f84326972836a20d9efe35eb1637f8,813dd61a591b393d
Normal-641.png,Normal,Normal/images/Normal-641.png,Normal/masks/Normal-641.png,True,G15618,train,ea8ad4de152e6f01208e4100de3ad0f8,917bce5a514e1987
Normal-6410.png,Normal,Normal/images/Normal-6410.png,Normal/masks/Normal-6410.png,True,G15619,train,6961f23d3e4a32812d684a93b48956fd,b979851e85c61e96
Normal-6411.png,Normal,Normal/images/Normal-6411.png,Normal/masks/Normal-6411.png,True,G15620,val,9e2d9f37b73eb1e113e97df7cd55fe1d,9847165a797a7962
Normal-6412.png,Normal,Normal/images/Normal-6412.png,Normal/masks/Normal-6412.png,True,G15621,val,c0da442bf9d1b0f61b1c7b37921a98c0,817f5618554f195d
Normal-6413.png,Normal,Normal/images/Normal-6413.png,Normal/masks/Normal-6413.png,True,G15622,train,b3bb49d99486a3fc1971d4f8c7d17d1e,d90d461f4a6c3ce9
Normal-6414.png,Normal,Normal/images/Normal-6414.png,Normal/masks/Normal-6414.png,True,G15623,val,53fbf7c665c09ccfbd59b854863227a4,d92d464f1b1c6d29
Normal-6415.png,Normal,Normal/images/Normal-6415.png,Normal/masks/Normal-6415.png,True,G15624,train,646c6ddc19923ec39d3b70b773eac37f,d928c61a59471f5d
Normal-6416.png,Normal,Normal/images/Normal-6416.png,Normal/masks/Normal-6416.png,True,G15625,train,ed269da27dfb09c32f407d5b4c0d9009,d83b971e171a1a1e
Normal-6417.png,Normal,Normal/images/Normal-6417.png,Normal/masks/Normal-6417.png,True,G15626,train,468000f6e76b9e9b0b3ea9bb4b31c127,897b87061f555e58
Normal-6418.png,Normal,Normal/images/Normal-6418.png,Normal/masks/Normal-6418.png,True,G15627,train,02c7174e3cab942dd30312435aec9354,d939061a3d687abc
Normal-6419.png,Normal,Normal/images/Normal-6419.png,Normal/masks/Normal-6419.png,True,G15628,test,7f88d836f7b61f0f9fc5f8338f3edbbc,d43d43430d0fac3f
Normal-642.png,Normal,Normal/images/Normal-642.png,Normal/masks/Normal-642.png,True,G15629,val,d55e11242afdfa8047a4530259de7b49,917cce1a99491ece
Normal-6420.png,Normal,Normal/images/Normal-6420.png,Normal/masks/Normal-6420.png,True,G15630,test,b2bec683a2c125cf704c8d2066245bf6,8138165e596b7b69
Normal-6421.png,Normal,Normal/images/Normal-6421.png,Normal/masks/Normal-6421.png,True,G15631,test,9a975f2537ade4c4807b78098e88904d,dd3cd25bd5435540
Normal-6422.png,Normal,Normal/images/Normal-6422.png,Normal/masks/Normal-6422.png,True,G15632,train,0bbff3c08ca5e60d6218df76c8302a28,817b86969a4d1a5f
Normal-6423.png,Normal,Normal/images/Normal-6423.png,Normal/masks/Normal-6423.png,True,G15633,train,daf69b9ba724d96a8d4d1e9bccaa7a2d,d53d424b1a465b1f
Normal-6424.png,Normal,Normal/images/Normal-6424.png,Normal/masks/Normal-6424.png,True,G15634,train,77d62d4ed1b1c18ac22d9ddcec51268b,985f56185b1b0e3b
Normal-6425.png,Normal,Normal/images/Normal-6425.png,Normal/masks/Normal-6425.png,True,G15635,train,553d554aa406811dca5c729523b284c7,c43d0b415d0b3f3d
Normal-6426.png,Normal,Normal/images/Normal-6426.png,Normal/masks/Normal-6426.png,True,G15636,train,e5e7085754a4be2f971df01123d1e4d4,d43f03434f0e2e6d
Normal-6427.png,Normal,Normal/images/Normal-6427.png,Normal/masks/Normal-6427.png,True,G15637,train,418e0ef8fb68f47747bdd585387d4ecf,9909161a5c7b596f
Normal-6428.png,Normal,Normal/images/Normal-6428.png,Normal/masks/Normal-6428.png,True,G15638,train,17e96a7e89904a41bcb560b58418f826,8878071e1f5b39cb
Normal-6429.png,Normal,Normal/images/Normal-6429.png,Normal/masks/Normal-6429.png,True,G15639,train,e3e1184968fecc0cbeb0c6262697e060,9169c716d54f1e51
Normal-643.png,Normal,Normal/images/Normal-643.png,Normal/masks/Normal-643.png,True,G15640,train,63cbb599bb318cd522667f6dafbb8248,85feaf0a090d4dc7
Normal-6430.png,Normal,Normal/images/Normal-6430.png,Normal/masks/Normal-6430.png,True,G15641,train,e3e456dd8c1ba64a85f668450164d329,91b946451b1b4f1f
Normal-6431.png,Normal,Normal/images/Normal-6431.png,Normal/masks/Normal-6431.png,True,G15642,train,03028777c0de33cf6d3365dd86bf3a65,dd3e520f0d441abd
Normal-6432.png,Normal,Normal/images/Normal-6432.png,Normal/masks/Normal-6432.png,True,G15643,test,3d994f3d50e31832ae9a6b8268d972f5,d909861e5d4f586b
Normal-6433.png,Normal,Normal/images/Normal-6433.png,Normal/masks/Normal-6433.png,True,G15644,train,2012a70afaec496d0f534680f1153663,897f861a1d5d194d
Normal-6434.png,Normal,Normal/images/Normal-6434.png,Normal/masks/Normal-6434.png,True,G15645,val,2f4b54c10ec747d0fd739e8c93bc64f6,d168861e595a1b5f
Normal-6435.png,Normal,Normal/images/Normal-6435.png,Normal/masks/Normal-6435.png,True,G15646,val,20dbd9251379cd407040d73a5ce0655c,d92e071b586db8b4
Normal-6436.png,Normal,Normal/images/Normal-6436.png,Normal/masks/Normal-6436.png,True,G15647,test,e459adae1866ca8ac4ca2f19f2d92a2f,997cc75b1a446b38
Normal-6437.png,Normal,Normal/images/Normal-6437.png,Normal/masks/Normal-6437.png,True,G15648,test,6ad81e5a2776e4de915ff8fcf095086e,9969065e1ade7a98
Normal-6438.png,Normal,Normal/images/Normal-6438.png,Normal/masks/Normal-6438.png,True,G15649,train,8bb8fcd8af14b4487e0d89c45c9b1f23,d93b161e1ca87939
Normal-6439.png,Normal,Normal/images/Normal-6439.png,Normal/masks/Normal-6439.png,True,G15650,train,b63acd32a9a836df4f28f0a8b5318587,d829c21f1f491f1d
Normal-644.png,Normal,Normal/images/Normal-644.png,Normal/masks/Normal-644.png,True,G15651,train,d7cec525fdf4806d413c8049816956cc,913e8f1e4d4d4cc6
Normal-6440.png,Normal,Normal/images/Normal-6440.png,Normal/masks/Normal-6440.png,True,G15652,train,6ad66eefb2955f9eec91d52980d21544,9979071e174e1b1b
Normal-6441.png,Normal,Normal/images/Normal-6441.png,Normal/masks/Normal-6441.png,True,G15653,train,ed56d07d9bf9d7ceed7deda4085e598e,95791e84195ae15f
Normal-6442.png,Normal,Normal/images/Normal-6442.png,Normal/masks/Normal-6442.png,True,G15654,train,af45a263e5177de2f7e7a5f7771f5107,9929861a555e79cd
Normal-6443.png,Normal,Normal/images/Normal-6443.png,Normal/masks/Normal-6443.png,True,G15655,train,56acd0a0a8374917f5b5981015e82e1c,9f780f141e1e1b1b
Normal-6444.png,Normal,Normal/images/Normal-6444.png,Normal/masks/Normal-6444.png,True,G15656,train,51b881fd97a02521b148471080f262dc,992c061a1f1d3d5f
Normal-6445.png,Normal,Normal/images/Normal-6445.png,Normal/masks/Normal-6445.png,True,G15657,train,6fdced2ef27c15862bf5ebb207e9fea9,d938064e186d7e79
Normal-6446.png,Normal,Normal/images/Normal-6446.png,Normal/masks/Normal-6446.png,True,G15658,train,d011137deb6016a56cc0ab5700c1e994,897e5618471b554f
Normal-6447.png,Normal,Normal/images/Normal-6447.png,Normal/masks/Normal-6447.png,True,G15659,train,0b19da4a368aa000aafbce8f66bf8f85,996c461b194f3db8
Normal-6448.png,Normal,Normal/images/Normal-6448.png,Normal/masks/Normal-6448.png,True,G15660,test,6ab92128081b50882cdf7fb8c317f33f,887b05861e187f5f
Normal-6449.png,Normal,Normal/images/Normal-6449.png,Normal/masks/Normal-6449.png,True,G15661,train,54a032b0702ea8b2766568db8b3f9d84,d93d04561e4e471f
Normal-645.png,Normal,Normal/images/Normal-645.png,Normal/masks/Normal-645.png,True,G15662,train,82166b741b23a2c72c81759b4ad4f4e4,91bdc69a99c525b4
Normal-6450.png,Normal,Normal/images/Normal-6450.png,Normal/masks/Normal-6450.png,True,G15663,val,f1086cf1008b585db2ab9624d068f061,d93d444b1b4f4a3c
Normal-6451.png,Normal,Normal/images/Normal-6451.png,Normal/masks/Normal-6451.png,True,G15664,train,acaf38265231b513f6939afc14064be8,dd3ac20f424f1d4e
Normal-6452.png,Normal,Normal/images/Normal-6452.png,Normal/masks/Normal-6452.png,True,G15665,train,d13f40d26a47fc472a42b24b6cdfe3ff,997f0656185c4e5b
Normal-6453.png,Normal,Normal/images/Normal-6453.png,Normal/masks/Normal-6453.png,True,G15666,train,58afdb26b042fc832bf86a8509d205da,99ad471b870f114f
Normal-6454.png,Normal,Normal/images/Normal-6454.png,Normal/masks/Normal-6454.png,True,G15667,val,ff62adb597f8fb42dad601af56d16ca6,dcaec319c71b084f
Normal-6455.png,Normal,Normal/images/Normal-6455.png,Normal/masks/Normal-6455.png,True,G15668,val,88585da1d0b08650630a400453aa3064,dd2ad20f4a6c1e6c
Normal-6456.png,Normal,Normal/images/Normal-6456.png,Normal/masks/Normal-6456.png,True,G15669,test,6d99567ccaaa766f2244d4ebc100ff13,897906865e87df1a
Normal-6457.png,Normal,Normal/images/Normal-6457.png,Normal/masks/Normal-6457.png,True,G15670,test,a5a2ee583856791615d5f0c1e1c59d4d,9979841e594d5bc9
Normal-6458.png,Normal,Normal/images/Normal-6458.png,Normal/masks/Normal-6458.png,True,G15671,train,9ee1cd9b5d19a07321457f6c154add0b,dc3c525b1f452f09
Normal-6459.png,Normal,Normal/images/Normal-6459.png,Normal/masks/Normal-6459.png,True,G15672,train,fec523045202e0f22058d7bf785a54e8,897987961e5e1f18
Normal-646.png,Normal,Normal/images/Normal-646.png,Normal/masks/Normal-646.png,True,G15673,train,deaf77a7f964aebe8073277c8fe227d8,912bca39d85919db
Normal-6460.png,Normal,Normal/images/Normal-6460.png,Normal/masks/Normal-6460.png,True,G15674,val,86a358e8052a1c2202bfe37eb8e3394f,d03d42571b0b2f3e
Normal-6461.png,Normal,Normal/images/Normal-6461.png,Normal/masks/Normal-6461.png,True,G15675,val,a3e12410137e220599a86beb1c89ba7b,d42f430b2c7f383c
Normal-6462.png,Normal,Normal/images/Normal-6462.png,Normal/masks/Normal-6462.png,True,G15676,val,df480b58e6e476de3a1c79d6d8d84070,816ec61a5bec78e1
Normal-6463.png,Normal,Normal/images/Normal-6463.png,Normal/masks/Normal-6463.png,True,G15677,train,9838639b49a0f3f7455aa3611ac5bb1b,d96c4619863bb8ec
Normal-6464.png,Normal,Normal/images/Normal-6464.png,Normal/masks/Normal-6464.png,True,G15678,train,cfaf9185a74cd72b8fa77d2e8969ca37,897b47961ad55e18
Normal-6465.png,Normal,Normal/images/Normal-6465.png,Normal/masks/Normal-6465.png,True,G15679,train,43b7101587ab2b42513682fdfa7000e2,897984c61b1d5f4b
Normal-6466.png,Normal,Normal/images/Normal-6466.png,Normal/masks/Normal-6466.png,True,G15680,test,c9c93b772de7ffe09ef5e5e3ea4c1ca6,d03d520b0f0f0f3f
Normal-6467.png,Normal,Normal/images/Normal-6467.png,Normal/masks/Normal-6467.png,True,G15681,train,2361b9b194acc2c76718ebba7a6f15a3,dd2cc65b19460a7d
Normal-6468.png,Normal,Normal/images/Normal-6468.png,Normal/masks/Normal-6468.png,True,G15682,train,3d94dc13802644d3d6d094cb92ede080,d97b05161e05575b
Normal-6469.png,Normal,Normal/images/Normal-6469.png,Normal/masks/Normal-6469.png,True,G15683,train,e9b502cf08648f9329578d050b069e53,b825871e4d0d1e5f
Normal-647.png,Normal,Normal/images/Normal-647.png,Normal/masks/Normal-647.png,True,G15684,train,c01276ffde181b817e265c0631a383f7,9169c67ae86469e5
Normal-6470.png,Normal,Normal/images/Normal-6470.png,Normal/masks/Normal-6470.png,True,G15685,train,4ea1ef379110be1adf3a177954992bb3,993f861ad96938e1
Normal-6471.png,Normal,Normal/images/Normal-6471.png,Normal/masks/Normal-6471.png,True,G15686,train,7f4661ec9f56f09fba561f019040c960,dd2ec34b0d0e692d
Normal-6472.png,Normal,Normal/images/Normal-6472.png,Normal/masks/Normal-6472.png,True,G15687,test,f7d7678ef51eeb2a697a718c1ba4783a,dc3b530b0c2a2daf
Normal-6473.png,Normal,Normal/images/Normal-6473.png,Normal/masks/Normal-6473.png,True,G15688,test,a18155cc55ba414bd5b1491eb73d47e7,d82d465928ed69f1
Normal-6474.png,Normal,Normal/images/Normal-6474.png,Normal/masks/Normal-6474.png,True,G15689,train,5b4a8e1857f7d69b2860632cf6696551,d97dc61a844e18d7
Normal-6475.png,Normal,Normal/images/Normal-6475.png,Normal/masks/Normal-6475.png,True,G15690,train,3b0a4b88ddb833047b713bfa57c0ed1e,896c075a0d1d7d7c
Normal-6476.png,Normal,Normal/images/Normal-6476.png,Normal/masks/Normal-6476.png,True,G15691,train,232c47612524d638ab5974f6a9f19712,8179861e1c5f3a79
Normal-6477.png,Normal,Normal/images/Normal-6477.png,Normal/masks/Normal-6477.png,True,G15692,train,a03d4544b95d6658c684f8a122bfe94b,d169c25f184e6e39
Normal-6478.png,Normal,Normal/images/Normal-6478.png,Normal/masks/Normal-6478.png,True,G15693,test,0d48a4d682af9ba4870312350f7fcbaa,dc2ec30b0cacea3d
Normal-6479.png,Normal,Normal/images/Normal-6479.png,Normal/masks/Normal-6479.png,True,G15694,train,f66d103a3d02a1b60ffeff484b22c647,882d861e0d5f1b5f
Normal-648.png,Normal,Normal/images/Normal-648.png,Normal/masks/Normal-648.png,True,G15695,train,d13f8c5ff68febe77bdb0139afe4c088,907dc61a8b1939dd
Normal-6480.png,Normal,Normal/images/Normal-6480.png,Normal/masks/Normal-6480.png,True,G15696,train,d6622182819cdfabb001c1f4a70a43f6,8879c7c65e10d55e
Normal-6481.png,Normal,Normal/images/Normal-6481.png,Normal/masks/Normal-6481.png,True,G15697,train,6059ca5b2d9b7b4fc033f232517f2775,cdbcc31d420f1a4f
Normal-6482.png,Normal,Normal/images/Normal-6482.png,Normal/masks/Normal-6482.png,True,G15698,train,71145175d522b873639044d7ed245b7f,d9e9455ec4571945
Normal-6483.png,Normal,Normal/images/Normal-6483.png,Normal/masks/Normal-6483.png,True,G15699,train,bb3f1a07a6c00ca5778381a652cca9cc,897957061e175e1b
Normal-6484.png,Normal,Normal/images/Normal-6484.png,Normal/masks/Normal-6484.png,True,G15700,train,0063f070d186200478baa5bcf313ac01,d43c4a0f2c2f9b93
Normal-6485.png,Normal,Normal/images/Normal-6485.png,Normal/masks/Normal-6485.png,True,G15701,val,01c753c95a8c6b923c8eab354645d700,953c6ac339949e78
Normal-6486.png,Normal,Normal/images/Normal-6486.png,Normal/masks/Normal-6486.png,True,G15702,test,fe7ab9b6d464ba10a82df5569e3478fd,dc2ac30f0c595d6e
Normal-6487.png,Normal,Normal/images/Normal-6487.png,Normal/masks/Normal-6487.png,True,G15703,test,196f9b5314eea0a0773bda8e7c1368ab,9878871e1bc55f1a
Normal-6488.png,Normal,Normal/images/Normal-6488.png,Normal/masks/Normal-6488.png,True,G15704,train,20533e81bb3ff0229d857d3f25701e6c,9969c710165f3c5b
Normal-6489.png,Normal,Normal/images/Normal-6489.png,Normal/masks/Normal-6489.png,True,G15705,train,4026e69d996c96099f0422c56bc59d7c,c87b578612174f1b
Normal-649.png,Normal,Normal/images/Normal-649.png,Normal/masks/Normal-649.png,True,G15706,train,e27dcd884aa20fc44b66b1e587219f26,d46ea72b4d4c68c3
Normal-6490.png,Normal,Normal/images/Normal-6490.png,Normal/masks/Normal-6490.png,True,G15707,train,8c353c071433003f10f0f72feba48145,c179c65656465e1b
Normal-6491.png,Normal,Normal/images/Normal-6491.png,Normal/masks/Normal-6491.png,True,G15708,val,7316c03ae4f3ff43b725aa5158a07769,dd2f421f1e4a1f18
Normal-6492.png,Normal,Normal/images/Normal-6492.png,Normal/masks/Normal-6492.png,True,G15709,val,0c3318fe98204b0a24ee02b444497526,986dc45ac14f3aad
Normal-6493.png,Normal,Normal/images/Normal-6493.png,Normal/masks/Normal-6493.png,True,G15710,train,859d585439da2f813fd81425544d6486,81781e5a195fe939
Normal-6494.png,Normal,Normal/images/Normal-6494.png,Normal/masks/Normal-6494.png,True,G15711,train,163ce93c64e56f203049fb0e7968fd07,9979871e1c871ed1
Normal-6495.png,Normal,Normal/images/Normal-6495.png,Normal/masks/Normal-6495.png,True,G15712,train,0861da8d95267e61cc3867df46bdc34d,dd28570f144b1f69
Normal-6496.png,Normal,Normal/images/Normal-6496.png,Normal/masks/Normal-6496.png,True,G15713,train,e2df152f30cce70afe84ee8f3079f930,81695e965a3ba9e8
Normal-6497.png,Normal,Normal/images/Normal-6497.png,Normal/masks/Normal-6497.png,True,G15714,train,feae5f0c13ac3cb88ab4cfd42e5362fe,d52ec25f09430f5d
Normal-6498.png,Normal,Normal/images/Normal-6498.png,Normal/masks/Normal-6498.png,True,G15715,train,1e53c91c132c1d6b42f67c7e4c13cea1,cc2f421f0a4f4f3c
Normal-6499.png,Normal,Normal/images/Normal-6499.png,Normal/masks/Normal-6499.png,True,G15716,train,4b75151b3b2a363bb947119863cb4592,902dc71a596f095f
Normal-65.png,Normal,Normal/images/Normal-65.png,Normal/masks/Normal-65.png,True,G15717,train,43d6e0a7c7dbebf469b6a31071ccf18b,912fc74b98c93b92
Normal-650.png,Normal,Normal/images/Normal-650.png,Normal/masks/Normal-650.png,True,G15718,train,4f00153297b3ad7d47af8f98731dd17b,c06bef48c05ad0df
Normal-6500.png,Normal,Normal/images/Normal-6500.png,Normal/masks/Normal-6500.png,True,G15719,train,a1c2849a3dd0d70f96c680195de7f197,dd2e461b09457dac
Normal-6501.png,Normal,Normal/images/Normal-6501.png,Normal/masks/Normal-6501.png,True,G15720,train,926319f5485fb917012eee634c8bd1d4,9168c41a5f5a395f
Normal-6502.png,Normal,Normal/images/Normal-6502.png,Normal/masks/Normal-6502.png,True,G15721,train,57e65f10cbb04290ad499e7d0cffbf98,dd3d821bc50f190f
Normal-6503.png,Normal,Normal/images/Normal-6503.png,Normal/masks/Normal-6503.png,True,G15722,train,0328c386f847e57d955d1f7f2f6a550d,997f86161e561a1b
Normal-6504.png,Normal,Normal/images/Normal-6504.png,Normal/masks/Normal-6504.png,True,G15723,val,fb98c00be0f6032f7aad677f45e174cd,99790612195f79b9
Normal-6505.png,Normal,Normal/images/Normal-6505.png,Normal/masks/Normal-6505.png,True,G15724,val,26844c7f2e152a19fc948577532963c5,d46c071d2d2e3e4e
Normal-6506.png,Normal,Normal/images/Normal-6506.png,Normal/masks/Normal-6506.png,True,G15725,val,c8158f64deb1c50158a24b8c41bdcc1c,d96d871e09451e5d
Normal-6507.png,Normal,Normal/images/Normal-6507.png,Normal/masks/Normal-6507.png,True,G15726,test,bb5202b29e0e012c4cb5f829b4536789,d97b44c21f014f1f
Normal-6508.png,Normal,Normal/images/Normal-6508.png,Normal/masks/Normal-6508.png,True,G15727,test,01a339327b2d4d17718ead24cd91b9c9,dd0a434b0e2f2e2f
Normal-6509.png,Normal,Normal/images/Normal-6509.png,Normal/masks/Normal-6509.png,True,G15728,test,5e07431dd51fd8e59b541cd29a886ea0,d93cc6191a4f394d
Normal-651.png,Normal,Normal/images/Normal-651.png,Normal/masks/Normal-651.png,True,G15729,train,a5f299491f3bc0020b0f5e8df7097d8f,917c873a893978e5
Normal-6510.png,Normal,Normal/images/Normal-6510.png,Normal/masks/Normal-6510.png,True,G15730,train,8e0149f66bdf7820bc94efd636601b41,d52fc0d31bc4425f
Normal-6511.png,Normal,Normal/images/Normal-6511.png,Normal/masks/Normal-6511.png,True,G15731,train,10604407cf49a799690538ab3d45ebbf,9caf031d831bc54f
Normal-6512.png,Normal,Normal/images/Normal-6512.png,Normal/masks/Normal-6512.png,True,G15732,train,40042be6de3ce477712a9951ababe1cd,9129861e1a975ebe
Normal-6513.png,Normal,Normal/images/Normal-6513.png,Normal/masks/Normal-6513.png,True,G15733,train,33f933d6961d738071d6f7e341c96dfb,d939c25b160f3b94
Normal-6514.png,Normal,Normal/images/Normal-6514.png,Normal/masks/Normal-6514.png,True,G15734,train,9b70e4b9cc94beed1703a066dab52111,d92fc61b0c0f39a5
Normal-6515.png,Normal,Normal/images/Normal-6515.png,Normal/masks/Normal-6515.png,True,G15735,train,d1591cdaeec006ecf362dc9d3529c05f,a97f86161c1f1a1d
Normal-6516.png,Normal,Normal/images/Normal-6516.png,Normal/masks/Normal-6516.png,True,G15736,train,bd781766efb9c008653b71ac199b4410,d42fc14b4e432f2e
Normal-6517.png,Normal,Normal/images/Normal-6517.png,Normal/masks/Normal-6517.png,True,G15737,val,2e81dd42f7030de47c5079b65528eb34,996c06185bed78b9
Normal-6518.png,Normal,Normal/images/Normal-6518.png,Normal/masks/Normal-6518.png,True,G15738,train,c1a29d8d736cf59be44a550d2db83fb0,cd2a02571bc21fdd
Normal-6519.png,Normal,Normal/images/Normal-6519.png,Normal/masks/Normal-6519.png,True,G15739,train,eb65cdc85a2f0eb644f22813e58b6268,992d061e1e5a7a7c
Normal-652.png,Normal,Normal/images/Normal-652.png,Normal/masks/Normal-652.png,True,G15740,train,a62ed7eb97c116438df977ccf89c48b4,d0edc31be86938c3
Normal-6520.png,Normal,Normal/images/Normal-6520.png,Normal/masks/Normal-6520.png,True,G15741,train,0357cab6da2acf107d6efc2e05dbb604,dc2c420fa8acbdf4
Normal-6521.png,Normal,Normal/images/Normal-6521.png,Normal/masks/Normal-6521.png,True,G15742,val,61d5be6ca54260c2a50c40491381596e,c03d431f5b6f2c38
Normal-6522.png,Normal,Normal/images/Normal-6522.png,Normal/masks/Normal-6522.png,True,G15743,train,e9a38e0aac0505178dd9148a0af9e2f8,9965461e461bd55b
Normal-6523.png,Normal,Normal/images/Normal-6523.png,Normal/masks/Normal-6523.png,True,G15744,train,c8b423c2ae34b3209820f1e3a98c525e,c9bb0218564f39cf
Normal-6524.png,Normal,Normal/images/Normal-6524.png,Normal/masks/Normal-6524.png,True,G15745,train,87070f54783939aedf5648f8601a9e95,8178dc1f581b784f
Normal-6525.png,Normal,Normal/images/Normal-6525.png,Normal/masks/Normal-6525.png,True,G15746,val,3ebe9986479b51f8f0651c58a6ccd4f4,897b05861e125edf
Normal-6526.png,Normal,Normal/images/Normal-6526.png,Normal/masks/Normal-6526.png,True,G15747,test,8e7d010663c0dc29e73d84dd6b2fe777,897f861a195d7a89
Normal-6527.png,Normal,Normal/images/Normal-6527.png,Normal/masks/Normal-6527.png,True,G15748,test,7c9c7ba0e885461ff1c8307f22e4006f,997b47161e545b1a
Normal-6528.png,Normal,Normal/images/Normal-6528.png,Normal/masks/Normal-6528.png,True,G15749,test,6ff1d22f287a910e04d446874ad81624,dc2f424b0aed3db0
Normal-6529.png,Normal,Normal/images/Normal-6529.png,Normal/masks/Normal-6529.png,True,G15750,train,2f12ad1e53d534d73805a19597fdd0f4,8879d5521fc5561b
Normal-653.png,Normal,Normal/images/Normal-653.png,Normal/masks/Normal-653.png,True,G15751,train,1495e6fd1e4c62a3566560e8ae1dfc39,942bc31b9c583cd7
Normal-6530.png,Normal,Normal/images/Normal-6530.png,Normal/masks/Normal-6530.png,True,G15752,train,230de62ff2a6d25b04984bdc3a8c64ac,992d061a5d4f38f9
Normal-6531.png,Normal,Normal/images/Normal-6531.png,Normal/masks/Normal-6531.png,True,G15753,train,4283cdf477dbc7c8d0e20300a5292d3a,d02dd2435b5e4979
Normal-6532.png,Normal,Normal/images/Normal-6532.png,Normal/masks/Normal-6532.png,True,G15754,train,579ee69f4e34c6a5407b6d012ea8c098,d92ad49717551a1e
Normal-6533.png,Normal,Normal/images/Normal-6533.png,Normal/masks/Normal-6533.png,True,G15755,train,ea62b5a6ce7513423b10c64b0cd0cfd7,f969861a415f514f
Normal-6534.png,Normal,Normal/images/Normal-6534.png,Normal/masks/Normal-6534.png,True,G15756,train,55211d1a79e83a6063fca26e145bb798,8c3e424f1d4d1d5d
Normal-6535.png,Normal,Normal/images/Normal-6535.png,Normal/masks/Normal-6535.png,True,G15757,train,37ca0ad3627c7fdbc0b815da249eba78,dc3c065b0b4e7e68
Normal-6536.png,Normal,Normal/images/Normal-6536.png,Normal/masks/Normal-6536.png,True,G15758,val,8236cc91175f4d304ad1529dab690321,c03a565f1f4c1b3c
Normal-6537.png,Normal,Normal/images/Normal-6537.png,Normal/masks/Normal-6537.png,True,G15759,train,e295ec11854c9e2da7d8b1691b6add34,817806575b7aec78
Normal-6538.png,Normal,Normal/images/Normal-6538.png,Normal/masks/Normal-6538.png,True,G15760,train,a4551c12bcf3b283d5b26a34791f42ca,992d865a1c6e68bd
Normal-6539.png,Normal,Normal/images/Normal-6539.png,Normal/masks/Normal-6539.png,True,G15761,train,7b92b1109f7a35f3ec153f1a4dfc20da,b81e861a1a7e3ab9
Normal-654.png,Normal,Normal/images/Normal-654.png,Normal/masks/Normal-654.png,True,G15762,train,1c0dcc2d90b7929b9417aae5033590f2,d17dc61e8d2d38c4
Normal-6540.png,Normal,Normal/images/Normal-6540.png,Normal/masks/Normal-6540.png,True,G15763,train,4a4f008cf7ec31565707b6920d9ebb2f,dd2e42131b4d1e5d
Normal-6541.png,Normal,Normal/images/Normal-6541.png,Normal/masks/Normal-6541.png,True,G15764,val,a9a94214644d3cebd6289028ff52ced6,9d2cc25b454eac4f
Normal-6542.png,Normal,Normal/images/Normal-6542.png,Normal/masks/Normal-6542.png,True,G15765,test,950d24d7c9e2986103ecd87c5006de75,817b861e5a5b3aca
Normal-6543.png,Normal,Normal/images/Normal-6543.png,Normal/masks/Normal-6543.png,True,G15766,val,ee54101db76c0de77e0cee2bf5d2c53f,887f05161f0d5e3b
Normal-6544.png,Normal,Normal/images/Normal-6544.png,Normal/masks/Normal-6544.png,True,G15767,train,0b1208e4f5a77ebb20e253a35e8cb5ba,dd2dc65307475954
Normal-6545.png,Normal,Normal/images/Normal-6545.png,Normal/masks/Normal-6545.png,True,G15768,val,952071aec985e5ee172b82338040aa35,d43d02c34f0fad4b
Normal-6546.png,Normal,Normal/images/Normal-6546.png,Normal/masks/Normal-6546.png,True,G15769,test,0f53c55b77b0f2a7285103d5b375ec27,896e06594f5a4d57
Normal-6547.png,Normal,Normal/images/Normal-6547.png,Normal/masks/Normal-6547.png,True,G15770,test,6e7d42df3b43c5b0ae9b1636bfe8197c,d979c51617461e4e
Normal-6548.png,Normal,Normal/images/Normal-6548.png,Normal/masks/Normal-6548.png,True,G15771,train,6e960002685a0d6b5c3f01de63d77827,8968871a59fd7ac1
Normal-6549.png,Normal,Normal/images/Normal-6549.png,Normal/masks/Normal-6549.png,True,G15772,train,5ddd51da3ec66afdc0a27b81082507ce,ddf9840f96851e15
Normal-655.png,Normal,Normal/images/Normal-655.png,Normal/masks/Normal-655.png,True,G15773,train,d0812bbe5f49c0138ce17a7bf553b6a5,906fc71e9e4139e4
Normal-6550.png,Normal,Normal/images/Normal-6550.png,Normal/masks/Normal-6550.png,True,G15774,train,22c22dc9674cc1b821c5de1aa6ac8964,dd2ec20b1dca199d
Normal-6551.png,Normal,Normal/images/Normal-6551.png,Normal/masks/Normal-6551.png,True,G15775,train,ae038c148fdd3092b57285e65b936991,996ec71957451a87
Normal-6552.png,Normal,Normal/images/Normal-6552.png,Normal/masks/Normal-6552.png,True,G15776,train,653186497ab59c2bfe5386353d44d791,91fd464f1b404f1d
Normal-6553.png,Normal,Normal/images/Normal-6553.png,Normal/masks/Normal-6553.png,True,G15777,train,49a4af2f9061c898051a9a1c99dc3205,997817165a1df83a
Normal-6554.png,Normal,Normal/images/Normal-6554.png,Normal/masks/Normal-6554.png,True,G15778,train,d5de339c4f069c5753aa6a23be57f956,d87f465a5c48786c
Normal-6555.png,Normal,Normal/images/Normal-6555.png,Normal/masks/Normal-6555.png,True,G15779,val,14dacc06493121723bab23ff14cf5c4b,993d871f194f1a91
Normal-6556.png,Normal,Normal/images/Normal-6556.png,Normal/masks/Normal-6556.png,True,G15780,train,9d9ccc098b47a77910e2e91ea34857d7,dd2cd20b1d482bbd
Normal-6557.png,Normal,Normal/images/Normal-6557.png,Normal/masks/Normal-6557.png,True,G15781,train,940dd7da70fe43b9cd7796223a3d44a9,dc2cd30b0c2e3df4
Normal-6558.png,Normal,Normal/images/Normal-6558.png,Normal/masks/Normal-6558.png,True,G15782,val,9aa7b46ba2892cc31a594108ddafbb68,d979c0171e570b1d
Normal-6559.png,Normal,Normal/images/Normal-6559.png,Normal/masks/Normal-6559.png,True,G15783,train,3b30500786ba30dbb47bf5639f8c6ac7,9d6c465a094f2f39
Normal-656.png,Normal,Normal/images/Normal-656.png,Normal/masks/Normal-656.png,True,G15784,train,e3979bf60436024c581ad1363cfe012c,d4f6c71bc98d2885
Normal-6560.png,Normal,Normal/images/Normal-6560.png,Normal/masks/Normal-6560.png,True,G15785,train,ba4de4dec819e99abb63b252231c98f8,dd3ac61f144d18cd
Normal-6561.png,Normal,Normal/images/Normal-6561.png,Normal/masks/Normal-6561.png,True,G15786,train,fdf99024c6a68caf028c7fc85076e152,cd2fc2c20f4c4f0f
Normal-6562.png,Normal,Normal/images/Normal-6562.png,Normal/masks/Normal-6562.png,True,G15787,train,8630b76a06a80924543246f5fa4a081a,817f861a1d4a3b5d
Normal-6563.png,Normal,Normal/images/Normal-6563.png,Normal/masks/Normal-6563.png,True,G15788,train,5d2d98c56b90f42c4caf08cb2f675f83,81785f861e195b5d
Normal-6564.png,Normal,Normal/images/Normal-6564.png,Normal/masks/Normal-6564.png,True,G15789,test,048b8fd2fa737705861816b2269eeefd,dd7dc25a17421b2c
Normal-6565.png,Normal,Normal/images/Normal-6565.png,Normal/masks/Normal-6565.png,True,G15790,test,c70a9a1fe6c51464c50c2e37eee3b2ae,c17cc24b184f2f3b
Normal-6566.png,Normal,Normal/images/Normal-6566.png,Normal/masks/Normal-6566.png,True,G15791,train,75c972ff60ca3b6bf9023085f76fd2de,dc3d42525f4e1c1d
Normal-6567.png,Normal,Normal/images/Normal-6567.png,Normal/masks/Normal-6567.png,True,G15792,train,f68eddc234034ba022fbbc5ec6bba4e1,d93d461b194c6db8
Normal-6568.png,Normal,Normal/images/Normal-6568.png,Normal/masks/Normal-6568.png,True,G15793,test,03119808afc309e935f3631ae68ab9de,dd2de2561f506a1c
Normal-6569.png,Normal,Normal/images/Normal-6569.png,Normal/masks/Normal-6569.png,True,G15794,train,113a83916a1eea6fe8f2c1d85f2d75cc,897d065e597da0f0
Normal-657.png,Normal,Normal/images/Normal-657.png,Normal/masks/Normal-657.png,True,G15795,train,8721bdf9a42376f97b5ed5b6908188e8,847ce31b8bcb4ccc
Normal-6570.png,Normal,Normal/images/Normal-6570.png,Normal/masks/Normal-6570.png,True,G15796,test,f04fcb290aa430c8ed3e0e02615079a6,91689e19195c79f5
Normal-6571.png,Normal,Normal/images/Normal-6571.png,Normal/masks/Normal-6571.png,True,G15797,train,0cbcf5fef7711f99876be935593e3525,992d0616071f5d5f
Normal-6572.png,Normal,Normal/images/Normal-6572.png,Normal/masks/Normal-6572.png,True,G15798,train,0473da249f92484b92da4ae1867d8fe3,fd4cc30b0d823fac
Normal-6573.png,Normal,Normal/images/Normal-6573.png,Normal/masks/Normal-6573.png,True,G15799,train,7eb40f50e71f8db43b41fb89a2f6f3e1,d96dd61fc45a5540
Normal-6574.png,Normal,Normal/images/Normal-6574.png,Normal/masks/Normal-6574.png,True,G15800,train,42da51366a5522f217b97a3a56bf2b73,d97a47861a4f2c3a
Normal-6575.png,Normal,Normal/images/Normal-6575.png,Normal/masks/Normal-6575.png,True,G15801,train,f3bf1645c7f2cb4186021c6372066bfa,d92f461a4d4b192f
Normal-6576.png,Normal,Normal/images/Normal-6576.png,Normal/masks/Normal-6576.png,True,G15802,val,33693350cdb35ffcbcdd1f39ec5e8ce4,9939861e1c5e7ae8
Normal-6577.png,Normal,Normal/images/Normal-6577.png,Normal/masks/Normal-6577.png,True,G15803,val,f99cb5151731ee969d437c0d5a29420d,997d871e18075a79
Normal-6578.png,Normal,Normal/images/Normal-6578.png,Normal/masks/Normal-6578.png,True,G15804,train,4e56df4f9897aae2aa805fdd39a7a9eb,dd2cd0d31b59c153
Normal-6579.png,Normal,Normal/images/Normal-6579.png,Normal/masks/Normal-6579.png,True,G15805,train,e77ac46968feb71e3a93ca6bc600931f,997d861a1d4d1b59
Normal-658.png,Normal,Normal/images/Normal-658.png,Normal/masks/Normal-658.png,True,G15806,val,4936e54b64f86232d08a1f26e76cef65,dd3ec21bc549186d
Normal-6580.png,Normal,Normal/images/Normal-6580.png,Normal/masks/Normal-6580.png,True,G15807,test,8fa8bcc05ce9b8fb19edfea1ce5efed6,806956df5b554a59
Normal-6581.png,Normal,Normal/images/Normal-6581.png,Normal/masks/Normal-6581.png,True,G15808,test,0089c3aed61bb3fe95c455d869cb04ab,996887121e4f7a1f
Normal-6582.png,Normal,Normal/images/Normal-6582.png,Normal/masks/Normal-6582.png,True,G15809,train,9b9ec84d50a8c3fec043b4bc70569335,9968851e5ed45af8
Normal-6583.png,Normal,Normal/images/Normal-6583.png,Normal/masks/Normal-6583.png,True,G15810,train,aa1c37c1f20f0aebeacb60b5bdccfb0c,d57c42031b4b2fba
Normal-6584.png,Normal,Normal/images/Normal-6584.png,Normal/masks/Normal-6584.png,True,G15811,train,a48b688bfead7b3d50e1a21e0722e869,dd29c21f960f1a59
Normal-6585.png,Normal,Normal/images/Normal-6585.png,Normal/masks/Normal-6585.png,True,G15812,train,dcb3938fd2017ce58772bf748315277b,cc3dd2430f534b0f
Normal-6586.png,Normal,Normal/images/Normal-6586.png,Normal/masks/Normal-6586.png,True,G15813,train,6508e1f77dc672b01629a61fd83cebeb,997d871e870e191d
Normal-6587.png,Normal,Normal/images/Normal-6587.png,Normal/masks/Normal-6587.png,True,G15814,train,703cc8934cb5201c983591b1a2295fbe,c436c32d0e4f2f2e
Normal-6588.png,Normal,Normal/images/Normal-6588.png,Normal/masks/Normal-6588.png,True,G15815,train,6d38ac651e5941d50ec99e093290bfee,dd6c475a09453a3d
Normal-6589.png,Normal,Normal/images/Normal-6589.png,Normal/masks/Normal-6589.png,True,G15816,train,c2e09e043f04e85e46912b1bfccb77c8,8869475a1f5d3a1d
Normal-659.png,Normal,Normal/images/Normal-659.png,Normal/masks/Normal-659.png,True,G15817,train,4f259a995cb965291cb7802085908a11,d91b061a186d79bd
Normal-6590.png,Normal,Normal/images/Normal-6590.png,Normal/masks/Normal-6590.png,True,G15818,train,03c24a492f83ccfa34e80b382ce01741,817b570e181f2d3b
Normal-6591.png,Normal,Normal/images/Normal-6591.png,Normal/masks/Normal-6591.png,True,G15819,train,5ec473e5af16d0c4d66aac24da46ac6b,f92d061a1d875b1d
Normal-6592.png,Normal,Normal/images/Normal-6592.png,Normal/masks/Normal-6592.png,True,G15820,train,18198c238424e19da8c9224e0a1a6513,986d871e071f5b0d
Normal-6593.png,Normal,Normal/images/Normal-6593.png,Normal/masks/Normal-6593.png,True,G15821,train,58f3111ea103fa7511a8eda567df4b1b,dc02c30c4e0bcfef
Normal-6594.png,Normal,Normal/images/Normal-6594.png,Normal/masks/Normal-6594.png,True,G15822,train,b137000046efcedac2ee5846622e599b,942e07597ee8b9c1
Normal-6595.png,Normal,Normal/images/Normal-6595.png,Normal/masks/Normal-6595.png,True,G15823,val,3441790fbf7967db08d0d0a6b428728d,817a1716185f7b78
Normal-6596.png,Normal,Normal/images/Normal-6596.png,Normal/masks/Normal-6596.png,True,G15824,val,41ee5a9b76b21a01b80a3c186342d936,dd2cd0c31b55c70b
Normal-6597.png,Normal,Normal/images/Normal-6597.png,Normal/masks/Normal-6597.png,True,G15825,test,a1de4b3e81f0caddc3d351cc76690b03,dd3d02c51b1a4d4f
Normal-6598.png,Normal,Normal/images/Normal-6598.png,Normal/masks/Normal-6598.png,True,G15826,train,87658d511cdec57cc36dd2c8fea1f14b,d96c061acd5e18ed
Normal-6599.png,Normal,Normal/images/Normal-6599.png,Normal/masks/Normal-6599.png,True,G15827,val,5b6239caa34602c855841f4da4ba27b4,997946161fc92d39
Normal-66.png,Normal,Normal/images/Normal-66.png,Normal/masks/Normal-66.png,True,G15828,test,b6c34522886a4acc0466a5bee8a33697,d07ac61a99ed3c94
Normal-660.png,Normal,Normal/images/Normal-660.png,Normal/masks/Normal-660.png,True,G15829,train,3bb72f834a0a2d16aa63fdc0bd7b2d4d,d43de33fc82d3884
Normal-6600.png,Normal,Normal/images/Normal-6600.png,Normal/masks/Normal-6600.png,True,G15830,train,65d3aafba35620aae231a340aa32057a,de35414a8f0b562f
Normal-6601.png,Normal,Normal/images/Normal-6601.png,Normal/masks/Normal-6601.png,True,G15831,val,119684643e7997961d1c49d3002ff872,dc24435a4f0e6bcd
Normal-6602.png,Normal,Normal/images/Normal-6602.png,Normal/masks/Normal-6602.png,True,G15832,test,93b14e63a128bb5ab2952263e6599ac7,997b051e1e0a5af9
Normal-6603.png,Normal,Normal/images/Normal-6603.png,Normal/masks/Normal-6603.png,True,G15833,train,d50b05814146c7871ca69f27f3b034e5,c93d464b1acc6db8
Normal-6604.png,Normal,Normal/images/Normal-6604.png,Normal/masks/Normal-6604.png,True,G15834,train,6c8667e49ed726a9cf1b79a8648b3268,8169861e15567ef9
Normal-6605.png,Normal,Normal/images/Normal-6605.png,Normal/masks/Normal-6605.png,True,G15835,train,e647e501318ca8f108c94050d868d6ff,8c0a420f5d1edbfa
Normal-6606.png,Normal,Normal/images/Normal-6606.png,Normal/masks/Normal-6606.png,True,G15836,train,0e84fd32f0a2c34f318c8a1fc5a3fa6d,c83ccf1b0b0a2fbc
Normal-6607.png,Normal,Normal/images/Normal-6607.png,Normal/masks/Normal-6607.png,True,G15837,train,3e9b30f1c79d6aeb5d431998b1c7f5df,882847175f1f7a6a
Normal-6608.png,Normal,Normal/images/Normal-6608.png,Normal/masks/Normal-6608.png,True,G15838,val,a42ce34a2fcca13f78557eaebe72eb48,917f96181d5a3a65
Normal-6609.png,Normal,Normal/images/Normal-6609.png,Normal/masks/Normal-6609.png,True,G15839,train,17fcebc63ee138613c10b12af79993b8,897b971e1c171b19
Normal-661.png,Normal,Normal/images/Normal-661.png,Normal/masks/Normal-661.png,True,G15840,train,ee66b916251e3b6aee3c6baa7a445fee,c46eab0dcd4e5865
Normal-6610.png,Normal,Normal/images/Normal-6610.png,Normal/masks/Normal-6610.png,True,G15841,train,6abd375f049d43faf1b0dc10d0246071,907d861aaf4abb4a
Normal-6611.png,Normal,Normal/images/Normal-6611.png,Normal/masks/Normal-6611.png,True,G15842,train,8696641ad0d2b03294150b4a63652dfb,d96f461f194f184c
Normal-6612.png,Normal,Normal/images/Normal-6612.png,Normal/masks/Normal-6612.png,True,G15843,train,0cc93f00ce05218d3ad3f6bbeff6e6ef,d13d4686531bcb0f
Normal-6613.png,Normal,Normal/images/Normal-6613.png,Normal/masks/Normal-6613.png,True,G15844,val,6531933f7606bd09df84636ec779c2f4,c939561d104f79f1
Normal-6614.png,Normal,Normal/images/Normal-6614.png,Normal/masks/Normal-6614.png,True,G15845,train,59c38fc7371b9cf0b7b4f04e8404e09e,dd3ec21353c71b11
Normal-6615.png,Normal,Normal/images/Normal-6615.png,Normal/masks/Normal-6615.png,True,G15846,test,a12f71bf4a1402de64881d92bef11af4,d97bc6161b1c4e1a
Normal-6616.png,Normal,Normal/images/Normal-6616.png,Normal/masks/Normal-6616.png,True,G15847,test,160d651fe2c3ecf4a6acb9e3b1e8d942,9b65545cca382b6d
Normal-6617.png,Normal,Normal/images/Normal-6617.png,Normal/masks/Normal-6617.png,True,G15848,test,90d670882a20f2a40521111856dff36d,dc2c825a494f3aed
Normal-6618.png,Normal,Normal/images/Normal-6618.png,Normal/masks/Normal-6618.png,True,G15849,train,df1a0c395b5e215a6c45b63cfefd0bd6,817a9618195f787b
Normal-6619.png,Normal,Normal/images/Normal-6619.png,Normal/masks/Normal-6619.png,True,G15850,train,d89a472bafd0f3e3363487358d7e33d9,d139468e1a1def19
Normal-662.png,Normal,Normal/images/Normal-662.png,Normal/masks/Normal-662.png,True,G15851,test,7b4b48091c803f6cace7cde068d55d24,916fc61ece0d18cd
Normal-6620.png,Normal,Normal/images/Normal-6620.png,Normal/masks/Normal-6620.png,True,G15852,train,3e5036671d22dd46281fa7a8e1025f44,dd2c420b4d6c5c77
Normal-6621.png,Normal,Normal/images/Normal-6621.png,Normal/masks/Normal-6621.png,True,G15853,train,359ed22c8ea1c66b4190733646cde64e,d928061e1ace7add
Normal-6622.png,Normal,Normal/images/Normal-6622.png,Normal/masks/Normal-6622.png,True,G15854,train,fdf2c422596534112f20ee31541efd88,dc3dc2431b4b0b3d
Normal-6623.png,Normal,Normal/images/Normal-6623.png,Normal/masks/Normal-6623.png,True,G15855,train,32b49b804d234fbcc12aa87d5e49dc31,8178165f197b39a9
Normal-6624.png,Normal,Normal/images/Normal-6624.png,Normal/masks/Normal-6624.png,True,G15856,train,7c33972b56669c7a8e1e68851733ced0,d92956875e550b1e
Normal-6625.png,Normal,Normal/images/Normal-6625.png,Normal/masks/Normal-6625.png,True,G15857,train,c233e3a2f046d49e0939ef4b0dbf95b9,dd28460f564e1b5d
Normal-6626.png,Normal,Normal/images/Normal-6626.png,Normal/masks/Normal-6626.png,True,G15858,train,6268c8a781db12af3eb8a75e97d6f48b,a80f061a497d5cdf
Normal-6627.png,Normal,Normal/images/Normal-6627.png,Normal/masks/Normal-6627.png,True,G15859,train,81adce22a6bb90ae74e18b166949b8b2,b94b861a4a595a5f
Normal-6628.png,Normal,Normal/images/Normal-6628.png,Normal/masks/Normal-6628.png,True,G15860,train,7cd6839c4b9e63edf66c01e863797385,81781e1c1b7e79a9
Normal-6629.png,Normal,Normal/images/Normal-6629.png,Normal/masks/Normal-6629.png,True,G15861,train,2325602ae35b56668c206205c0b5840a,d92d421b0d0f3afc
Normal-663.png,Normal,Normal/images/Normal-663.png,Normal/masks/Normal-663.png,True,G15862,train,dfde41d7c2d1bdd319c4509c9c9cc393,d43ce61b9f4b1a44
Normal-6630.png,Normal,Normal/images/Normal-6630.png,Normal/masks/Normal-6630.png,True,G15863,val,e36e9e8f11a2a1d262ea03a7d55a427b,d92fc61b194d1b38
Normal-6631.png,Normal,Normal/images/Normal-6631.png,Normal/masks/Normal-6631.png,True,G15864,val,aeb44b231685bff8b9516806c3fefd43,d928461b0e4f1b5f
Normal-6632.png,Normal,Normal/images/Normal-6632.png,Normal/masks/Normal-6632.png,True,G15865,test,51c608dfd56c51da69f9d9a3af841f6f,dc2e430f4cecb8e4
Normal-6633.png,Normal,Normal/images/Normal-6633.png,Normal/masks/Normal-6633.png,True,G15866,train,41d6d3bdc501ad71d466ed8f40ae3035,982d861a5d6d78ec
Normal-6634.png,Normal,Normal/images/Normal-6634.png,Normal/masks/Normal-6634.png,True,G15867,val,a8e43e86ea6ed89ed8631e46666953c9,996cc71b494e3a4d
Normal-6635.png,Normal,Normal/images/Normal-6635.png,Normal/masks/Normal-6635.png,True,G15868,train,3af8ddebaec158e26d70be3ad67d6d52,d92d061e1ece1af8
Normal-6636.png,Normal,Normal/images/Normal-6636.png,Normal/masks/Normal-6636.png,True,G15869,test,7f65552d8951c1450edd3a9290927e8c,8979071a594f4e79
Normal-6637.png,Normal,Normal/images/Normal-6637.png,Normal/masks/Normal-6637.png,True,G15870,train,38f9045215e874bfdb26fade45021e3f,9968870213d65f5f
Normal-6638.png,Normal,Normal/images/Normal-6638.png,Normal/masks/Normal-6638.png,True,G15871,train,af2e5c67670800fa61d233ca0558a699,997947063ad73e90
Normal-6639.png,Normal,Normal/images/Normal-6639.png,Normal/masks/Normal-6639.png,True,G15872,train,2018059f5e1ae4d28251822c31a88b6e,811b161e5be9eaa5
Normal-664.png,Normal,Normal/images/Normal-664.png,Normal/masks/Normal-664.png,True,G15873,test,211c51cb5e32a1cd70be43104ba59aee,d47fa31b8d0c2cc6
Normal-6640.png,Normal,Normal/images/Normal-6640.png,Normal/masks/Normal-6640.png,True,G15874,train,562a9952796cad35c3a66026514f3221,992f864b1f065b1d
Normal-6641.png,Normal,Normal/images/Normal-6641.png,Normal/masks/Normal-6641.png,True,G15875,train,748690a4f448c4287e66e1582c2d8b98,d56e460b0b2f3b2c
Normal-6642.png,Normal,Normal/images/Normal-6642.png,Normal/masks/Normal-6642.png,True,G15876,train,79458290587e5ceb48a7419154ac1f0b,d42e035b4ceca8fc
Normal-6643.png,Normal,Normal/images/Normal-6643.png,Normal/masks/Normal-6643.png,True,G15877,train,c602cbd11bd4b0cd55b582eb20e6e19c,8139465e585f4f39
Normal-6644.png,Normal,Normal/images/Normal-6644.png,Normal/masks/Normal-6644.png,True,G15878,train,c525ef3a678fffff963a6083038d0a9e,895d821b1d5e7ba8
Normal-6645.png,Normal,Normal/images/Normal-6645.png,Normal/masks/Normal-6645.png,True,G15879,val,f26762085fa932820f161e3377b2ad16,c13cce0a19467ef9
Normal-6646.png,Normal,Normal/images/Normal-6646.png,Normal/masks/Normal-6646.png,True,G15880,train,274bd43aaefbb5da918559be2468ca38,997dc71a104f1e4b
Normal-6647.png,Normal,Normal/images/Normal-6647.png,Normal/masks/Normal-6647.png,True,G15881,val,64afb7c6ac3195a51bb467922a7176ef,997985061e87575e
Normal-6648.png,Normal,Normal/images/Normal-6648.png,Normal/masks/Normal-6648.png,True,G15882,train,9173cee29ed77fe12d7f5078af19295b,882c475e574f78e8
Normal-6649.png,Normal,Normal/images/Normal-6649.png,Normal/masks/Normal-6649.png,True,G15883,train,85549786b961b08e5bf5a26beb7965e6,d53b425f1b491f09
Normal-665.png,Normal,Normal/images/Normal-665.png,Normal/masks/Normal-665.png,True,G15884,train,a70888e500f10ad46d14958920a76f2c,d46fa30b0b0b1cee
Normal-6650.png,Normal,Normal/images/Normal-6650.png,Normal/masks/Normal-6650.png,True,G15885,train,9c34bf6f0ad0973f9d336cf2d9bd3a2a,dd2cd60f5e442a5c
Normal-6651.png,Normal,Normal/images/Normal-6651.png,Normal/masks/Normal-6651.png,True,G15886,val,ec6167223619e354ce1fffc6d0644ef3,dd3e425dc2431b4b
Normal-6652.png,Normal,Normal/images/Normal-6652.png,Normal/masks/Normal-6652.png,True,G15887,test,9cbba919cfdb13589a38bcf1dddaabac,917b56124b5f194d
Normal-6653.png,Normal,Normal/images/Normal-6653.png,Normal/masks/Normal-6653.png,True,G15888,test,2ff683746c85987a3ff5d0cbdcfe8258,d12fc7161b471e1a
Normal-6654.png,Normal,Normal/images/Normal-6654.png,Normal/masks/Normal-6654.png,True,G15889,train,cca66fd5b9f230f54fed11873455787c,997d841e0c5e1ae7
Normal-6655.png,Normal,Normal/images/Normal-6655.png,Normal/masks/Normal-6655.png,True,G15890,train,dbdac0eea12db343adc964ac51943ac4,993d851b1e4e1abc
Normal-6656.png,Normal,Normal/images/Normal-6656.png,Normal/masks/Normal-6656.png,True,G15891,train,eb4221707c6862b347e34b2e754c49ac,917b879696471e13
Normal-6657.png,Normal,Normal/images/Normal-6657.png,Normal/masks/Normal-6657.png,True,G15892,train,0dfb5fff949f3ecd34f9bba2482c9398,b94f061a4b7878e9
Normal-6658.png,Normal,Normal/images/Normal-6658.png,Normal/masks/Normal-6658.png,True,G15893,train,1b4b8e8b0934df7328f6c2e77570f8cf,993dc60311872fbb
Normal-6659.png,Normal,Normal/images/Normal-6659.png,Normal/masks/Normal-6659.png,True,G15894,train,8a076e38753b6139abdf56828e7286bf,817916161b5d5e7a
Normal-666.png,Normal,Normal/images/Normal-666.png,Normal/masks/Normal-666.png,True,G15895,train,2b083101f4b4134793497bf9ba4dd9d9,81a1c67af86b3969
Normal-6660.png,Normal,Normal/images/Normal-6660.png,Normal/masks/Normal-6660.png,True,G15896,train,e812f74b4f3e8e33d4e5d578c3bbc5b1,9179c65a190f7939
Normal-6661.png,Normal,Normal/images/Normal-6661.png,Normal/masks/Normal-6661.png,True,G15897,train,4fe25cd63ffec1ad997e8d671070807d,802f571f3d2d2d34
Normal-6662.png,Normal,Normal/images/Normal-6662.png,Normal/masks/Normal-6662.png,True,G15898,train,9218092b3cf5261d0b709e34cb6aceeb,d97947851a17471e
Normal-6663.png,Normal,Normal/images/Normal-6663.png,Normal/masks/Normal-6663.png,True,G15899,train,00537f4781906e7c596ee776616eeb28,c86d420c486b7f6f
Normal-6664.png,Normal,Normal/images/Normal-6664.png,Normal/masks/Normal-6664.png,True,G15900,train,a2b5fac6a99356a2f133ee6fa210f23e,9978870b1d4f7a38
Normal-6665.png,Normal,Normal/images/Normal-6665.png,Normal/masks/Normal-6665.png,True,G15901,train,15c14cd4fb1ba368ff6ba5e04f83ac70,c93ec3165a4d1e59
Normal-6666.png,Normal,Normal/images/Normal-6666.png,Normal/masks/Normal-6666.png,True,G15902,train,04024e1a12d7d67cce6153c6d1f5bf92,d42d42134b4f1a7f
Normal-6667.png,Normal,Normal/images/Normal-6667.png,Normal/masks/Normal-6667.png,True,G15903,train,923166a4e6f6e59323f8a027c1fa594d,cc748f0f1d4d5c51
Normal-6668.png,Normal,Normal/images/Normal-6668.png,Normal/masks/Normal-6668.png,True,G15904,val,23b28be0fe0c87349a0e8451a579bcfc,dd3a421b4a6d3ce8
Normal-6669.png,Normal,Normal/images/Normal-6669.png,Normal/masks/Normal-6669.png,True,G15905,val,667012a53c76702925de2e54bb7e8d86,887b7fa7c6165a01
Normal-667.png,Normal,Normal/images/Normal-667.png,Normal/masks/Normal-667.png,True,G15906,test,e26a60b1d94ec927f26a30e048bcc023,d07c279b2b857ac4
Normal-6670.png,Normal,Normal/images/Normal-6670.png,Normal/masks/Normal-6670.png,True,G15907,val,18a8c9a351f51f0916c73c7d3751974d,d97dc65ac45a7848
Normal-6671.png,Normal,Normal/images/Normal-6671.png,Normal/masks/Normal-6671.png,True,G15908,test,f5a2cafec0cd980cab2d4cbaf056a58b,d178465a4b4f391d
Normal-6672.png,Normal,Normal/images/Normal-6672.png,Normal/masks/Normal-6672.png,True,G15909,train,a30a9fe17c21bb7dbe94f7101b5ba46c,99a1ce5e5a4c1aab
Normal-6673.png,Normal,Normal/images/Normal-6673.png,Normal/masks/Normal-6673.png,True,G15910,test,19bfce897480bd26bc2eb429c6d3eb3b,993c46471a996d3d
Normal-6674.png,Normal,Normal/images/Normal-6674.png,Normal/masks/Normal-6674.png,True,G15911,train,f8d6892047082afceb047b9025a3e053,992dc71bc44f5c61
Normal-6675.png,Normal,Normal/images/Normal-6675.png,Normal/masks/Normal-6675.png,True,G15912,train,861f0aec1ac62190358e25bfe364ab46,913f061e5e1e585d
Normal-6676.png,Normal,Normal/images/Normal-6676.png,Normal/masks/Normal-6676.png,True,G15913,train,546cd3cf0ebe6b5d3b43f550f9eb2120,d93fd61e145a1945
Normal-6677.png,Normal,Normal/images/Normal-6677.png,Normal/masks/Normal-6677.png,True,G15914,val,40283a8bc08196e03725e7ef2a5c82e5,cc3cd2c31bc2c31f
Normal-6678.png,Normal,Normal/images/Normal-6678.png,Normal/masks/Normal-6678.png,True,G15915,train,e0b84800cb97c1c22afb486d168df60b,dd2cd20b0fc20bbd
Normal-6679.png,Normal,Normal/images/Normal-6679.png,Normal/masks/Normal-6679.png,True,G15916,train,c3705e3cac83cc21c3d952eeb3a9ce34,d42cd31b1f6a455a
Normal-668.png,Normal,Normal/images/Normal-668.png,Normal/masks/Normal-668.png,True,G15917,train,4894058f3ddaa5c9e07a39bf7ab1b065,d43ec71f2d0d2c2c
Normal-6680.png,Normal,Normal/images/Normal-6680.png,Normal/masks/Normal-6680.png,True,G15918,val,b5e9f63be35eaa4e708dc645fc2677b7,dc0e4b4d0caef5a2
Normal-6681.png,Normal,Normal/images/Normal-6681.png,Normal/masks/Normal-6681.png,True,G15919,val,b297c8c9a76bf4e179337c60b637577d,d97dc2125a4f5a1a
Normal-6682.png,Normal,Normal/images/Normal-6682.png,Normal/masks/Normal-6682.png,True,G15920,test,55435938ad4c1382c3245351eebd9cef,897c071a1f4e5a6d
Normal-6683.png,Normal,Normal/images/Normal-6683.png,Normal/masks/Normal-6683.png,True,G15921,train,4ce0254b282b02ca06035455ab06691e,897e061a5d1d1f0f
Normal-6684.png,Normal,Normal/images/Normal-6684.png,Normal/masks/Normal-6684.png,True,G15922,train,67e65c3b7b8001f96a2696567507e665,d13d06875a1fca1b
Normal-6685.png,Normal,Normal/images/Normal-6685.png,Normal/masks/Normal-6685.png,True,G15923,train,b59a8203682de667bb4be89b1dda7a28,996c92439e5bc64b
Normal-6686.png,Normal,Normal/images/Normal-6686.png,Normal/masks/Normal-6686.png,True,G15924,train,c9355242f79a314e3e42d0907c00799b,9979c656415f5a4a
Normal-6687.png,Normal,Normal/images/Normal-6687.png,Normal/masks/Normal-6687.png,True,G15925,train,d8e071229e27491435a593aedac174fa,d46c43430f0fad3d
Normal-6688.png,Normal,Normal/images/Normal-6688.png,Normal/masks/Normal-6688.png,True,G15926,train,aff0b17f34246484abe5ca626267c320,cc344b0a1ced6f74
Normal-6689.png,Normal,Normal/images/Normal-6689.png,Normal/masks/Normal-6689.png,True,G15927,train,3522285b6651a70c7e2e199efe63ac4c,d935461e471b1d4b
Normal-669.png,Normal,Normal/images/Normal-669.png,Normal/masks/Normal-669.png,True,G15928,test,258f48527fd86926ea924370a9d78149,91bfc61e950135c7
Normal-6690.png,Normal,Normal/images/Normal-6690.png,Normal/masks/Normal-6690.png,True,G15929,train,918c9c91a5490acffa308708c083df4a,91690e161b5f7978
Normal-6691.png,Normal,Normal/images/Normal-6691.png,Normal/masks/Normal-6691.png,True,G15930,test,ca47bdaeeb8198444a146cb4f058dbfe,957886821f8f4e1f
Normal-6692.png,Normal,Normal/images/Normal-6692.png,Normal/masks/Normal-6692.png,True,G15931,train,ee01c7ef6b1ae98ebf296f6839407118,996a061e1ecc3be9
Normal-6693.png,Normal,Normal/images/Normal-6693.png,Normal/masks/Normal-6693.png,True,G15932,train,ee7e93f3679b3b3d6de5f75015b590a0,d929c65a1f524b1d
Normal-6694.png,Normal,Normal/images/Normal-6694.png,Normal/masks/Normal-6694.png,True,G15933,test,25a49a1409c5b1522741a095476111eb,cc2dc21f1a0e2fbc
Normal-6695.png,Normal,Normal/images/Normal-6695.png,Normal/masks/Normal-6695.png,True,G15934,train,67bacf5d2ac9f07dc10573d25fee2350,91796e8e75c4825b
Normal-6696.png,Normal,Normal/images/Normal-6696.png,Normal/masks/Normal-6696.png,True,G15935,train,b4b6f44a3933e7bfcfc46eab17a0c6c0,cc3c434b0a8baf3d
Normal-6697.png,Normal,Normal/images/Normal-6697.png,Normal/masks/Normal-6697.png,True,G15936,train,8fd6daf848432cbaa59cac62be38403e,9875061e151e1f5f
Normal-6698.png,Normal,Normal/images/Normal-6698.png,Normal/masks/Normal-6698.png,True,G15937,train,e3f3cec611a63e87e00df435484075e0,d03c87191dce69f8
Normal-6699.png,Normal,Normal/images/Normal-6699.png,Normal/masks/Normal-6699.png,True,G15938,train,8961c8fb41ff9dcc2076ec07dee49b91,986d461a595d5c5e
Normal-67.png,Normal,Normal/images/Normal-67.png,Normal/masks/Normal-67.png,True,G15939,train,5aaf648a6edabac71aaa7e38fb737ec4,907ce73ead0f3824
Normal-670.png,Normal,Normal/images/Normal-670.png,Normal/masks/Normal-670.png,True,G15940,val,d620b5213e7530a42017018156456414,946ec71ecd4c0ccd
Normal-6700.png,Normal,Normal/images/Normal-6700.png,Normal/masks/Normal-6700.png,True,G15941,train,c08a0e17934ac58215448670ee56318f,91281e7a69e8cece
Normal-6701.png,Normal,Normal/images/Normal-6701.png,Normal/masks/Normal-6701.png,True,G15942,train,f3ec9332e1940d6ffe46565f6ece2764,8979871a106e3fb9
Normal-6702.png,Normal,Normal/images/Normal-6702.png,Normal/masks/Normal-6702.png,True,G15943,train,9c45687a0b8d513f37c5c917d2bb8fe3,dd2fc04f0a4d4a1f
Normal-6703.png,Normal,Normal/images/Normal-6703.png,Normal/masks/Normal-6703.png,True,G15944,train,080779eb135e4fd0a3255722ec5b1766,9c2883c31b5dca5f
Normal-6704.png,Normal,Normal/images/Normal-6704.png,Normal/masks/Normal-6704.png,True,G15945,test,ffcee029c56d2a58a59dd9ecee652d77,99fed65b424f1882
Normal-6705.png,Normal,Normal/images/Normal-6705.png,Normal/masks/Normal-6705.png,True,G15946,val,3882e406151ba2c95194b29986903deb,c02d4bd23fd22f89
Normal-6706.png,Normal,Normal/images/Normal-6706.png,Normal/masks/Normal-6706.png,True,G15947,test,34befdb452905f17baa7598925016c44,d43d53431d0b8e1f
Normal-6707.png,Normal,Normal/images/Normal-6707.png,Normal/masks/Normal-6707.png,True,G15948,val,c25dfdd1a3ad30d4ba7c6215ff8c9590,dd2e434f086e39f0
Normal-6708.png,Normal,Normal/images/Normal-6708.png,Normal/masks/Normal-6708.png,True,G15949,train,f66cbade1638b64f5fd015f95187ba64,d92b87171d5d1919
Normal-6709.png,Normal,Normal/images/Normal-6709.png,Normal/masks/Normal-6709.png,True,G15950,train,fb666e83f70a29f23dae5cc94ba5399b,9168c75e5b851a5b
Normal-671.png,Normal,Normal/images/Normal-671.png,Normal/masks/Normal-671.png,True,G15951,train,931d76e2c74a0346643a71ab6d836512,913dc61a9cc13d97
Normal-6710.png,Normal,Normal/images/Normal-6710.png,Normal/masks/Normal-6710.png,True,G15952,train,ec8b24d84c899c957d6fa7aa778e22d2,916946161b0b7f3b
Normal-6711.png,Normal,Normal/images/Normal-6711.png,Normal/masks/Normal-6711.png,True,G15953,test,f9fdcf9337b8ecee753cf0136da77fb6,9978871a1b457d59
Normal-6712.png,Normal,Normal/images/Normal-6712.png,Normal/masks/Normal-6712.png,True,G15954,train,9e226283c130244f58eff15ed63565ba,917846461a4f6f3d
Normal-6713.png,Normal,Normal/images/Normal-6713.png,Normal/masks/Normal-6713.png,True,G15955,train,a8554ada3c617f0480198426193ab7ce,d96f825b1f551a44
Normal-6714.png,Normal,Normal/images/Normal-6714.png,Normal/masks/Normal-6714.png,True,G15956,train,d93343f1ed57876dcc761e710227f398,c87b07061e157b5b
Normal-6715.png,Normal,Normal/images/Normal-6715.png,Normal/masks/Normal-6715.png,True,G15957,train,1ae18573798d56724834cf9638ddf304,8179c7961a5b5959
Normal-6716.png,Normal,Normal/images/Normal-6716.png,Normal/masks/Normal-6716.png,True,G15958,train,9c7207a76c2e65fb62f914a9ed4b7250,d43d431b0f4b1a4f
Normal-6717.png,Normal,Normal/images/Normal-6717.png,Normal/masks/Normal-6717.png,True,G15959,train,f8c6a3bfe55b6b146c521e40bef64cae,8969061e1ecb39dd
Normal-6718.png,Normal,Normal/images/Normal-6718.png,Normal/masks/Normal-6718.png,True,G15960,val,b905187a3d67e6ba73b30ed73e32eed0,897c871a594d5ad9
Normal-6719.png,Normal,Normal/images/Normal-6719.png,Normal/masks/Normal-6719.png,True,G15961,train,7f50f80042fbc826545a1e42277b0726,9979861e1ac54fb8
Normal-672.png,Normal,Normal/images/Normal-672.png,Normal/masks/Normal-672.png,True,G15962,test,ce689600ad24e37819cf85b5e923014e,9560c678cd2d6dc6
Normal-6720.png,Normal,Normal/images/Normal-6720.png,Normal/masks/Normal-6720.png,True,G15963,train,1e57e558977d07bb372eba8945bcafaf,8d3cc35b164e1f58
Normal-6721.png,Normal,Normal/images/Normal-6721.png,Normal/masks/Normal-6721.png,True,G15964,train,68f30a940701f25ff6f08878c09845dc,997b05105b6d787a
Normal-6722.png,Normal,Normal/images/Normal-6722.png,Normal/masks/Normal-6722.png,True,G15965,train,2568d3dd193591478b55f427ed8230ea,d43cab43cf0abc4a
Normal-6723.png,Normal,Normal/images/Normal-6723.png,Normal/masks/Normal-6723.png,True,G15966,train,c9da9c46ec60651085ac916d5c6eea4d,985f121a093ab7b7
Normal-6724.png,Normal,Normal/images/Normal-6724.png,Normal/masks/Normal-6724.png,True,G15967,train,abc27d0ca6b2d3fefde7b2372d0c5a88,d829c6575f461e1a
Normal-6725.png,Normal,Normal/images/Normal-6725.png,Normal/masks/Normal-6725.png,True,G15968,train,a94d6602021f41e0121434bb925a6135,dc2d42574b025f1f
Normal-6726.png,Normal,Normal/images/Normal-6726.png,Normal/masks/Normal-6726.png,True,G15969,val,8fc4e7d8883abebd9e567271969473c5,817916161e5f5e3c
Normal-6727.png,Normal,Normal/images/Normal-6727.png,Normal/masks/Normal-6727.png,True,G15970,val,610049017798d60263bd58bbe620fc66,c82a97031c1e5f5f
Normal-6728.png,Normal,Normal/images/Normal-6728.png,Normal/masks/Normal-6728.png,True,G15971,test,4aa6568ec469e66497d9254b5651eabc,d968c61e184f3ba9
Normal-6729.png,Normal,Normal/images/Normal-6729.png,Normal/masks/Normal-6729.png,True,G15972,train,baf16ceaac85812079bb5a296150b9cd,d93b56031b554f1a
Normal-673.png,Normal,Normal/images/Normal-673.png,Normal/masks/Normal-673.png,True,G15973,train,b1de634607b7e29435d90839aa67da9e,d4d6c65acda560c6
Normal-6730.png,Normal,Normal/images/Normal-6730.png,Normal/masks/Normal-6730.png,True,G15974,train,4031e9b1a20f7faf9d4fe6471b5a60e3,dd68d30b0d0c0fbd
Normal-6731.png,Normal,Normal/images/Normal-6731.png,Normal/masks/Normal-6731.png,True,G15975,train,d4ab203e71b17b3af77a9fcbe9eb108f,9c4cc6da0aedaab8
Normal-6732.png,Normal,Normal/images/Normal-6732.png,Normal/masks/Normal-6732.png,True,G15976,train,1a3a736f4e948b36edbc1a7cc26e79fc,d93cc24b1c0e2fbc
Normal-6733.png,Normal,Normal/images/Normal-6733.png,Normal/masks/Normal-6733.png,True,G15977,train,c7cf1c542f8416559a9c8e6b6390fca9,897d875a1b4c6ab8
Normal-6734.png,Normal,Normal/images/Normal-6734.png,Normal/masks/Normal-6734.png,True,G15978,train,d4502cceaa4688fca609fd90624b4bde,897b041e1a5b595f
Normal-6735.png,Normal,Normal/images/Normal-6735.png,Normal/masks/Normal-6735.png,True,G15979,train,906fda0d9b792e67a93b49fc33589be2,887d87571a0c7f1a
Normal-6736.png,Normal,Normal/images/Normal-6736.png,Normal/masks/Normal-6736.png,True,G15980,train,129bd3c08db7ce767eacf0ab46075e1c,dd3d42c30b0cee2b
Normal-6737.png,Normal,Normal/images/Normal-6737.png,Normal/masks/Normal-6737.png,True,G15981,train,017b8fbfcf9f673ffa454d2d1da0a0af,dd2cd20b97461b4d
Normal-6738.png,Normal,Normal/images/Normal-6738.png,Normal/masks/Normal-6738.png,True,G15982,train,9b71eaf64df49c4140476976c2fa419a,dd2c435b454e154f
Normal-6739.png,Normal,Normal/images/Normal-6739.png,Normal/masks/Normal-6739.png,True,G15983,val,8a285ca5b44e39c66a8480aa54eeb24e,dd3cc24b1c431f3c
Normal-674.png,Normal,Normal/images/Normal-674.png,Normal/masks/Normal-674.png,True,G15984,val,fc52bc6475a279cd60421cdc30853f39,996fc71a491d384d
Normal-6740.png,Normal,Normal/images/Normal-6740.png,Normal/masks/Normal-6740.png,True,G15985,test,0362a27b5e1fc9288e8ef41106792b4b,d94dc61a4d1d194f
Normal-6741.png,Normal,Normal/images/Normal-6741.png,Normal/masks/Normal-6741.png,True,G15986,val,6021d0b5c284965e0456ff9503b958e5,8178161a396dbbb9
Normal-6742.png,Normal,Normal/images/Normal-6742.png,Normal/masks/Normal-6742.png,True,G15987,test,b8d1a161b155f9aafa653aa5ab9e590c,997fc61a804b3ba9
Normal-6743.png,Normal,Normal/images/Normal-6743.png,Normal/masks/Normal-6743.png,True,G15988,test,05dc14075e8f0ec37c3a30a573b39244,d52f42575b405f1a
Normal-6744.png,Normal,Normal/images/Normal-6744.png,Normal/masks/Normal-6744.png,True,G15989,train,d9f1daee0fdf3930c6fb0b2f5f2d3eb8,99681616195f5f59
Normal-6745.png,Normal,Normal/images/Normal-6745.png,Normal/masks/Normal-6745.png,True,G15990,train,0766638a43988885bdd94d348dd389f0,d87dc25712462b1f
Normal-6746.png,Normal,Normal/images/Normal-6746.png,Normal/masks/Normal-6746.png,True,G15991,train,5966e72576f6f0ee5a5951ddbdd93cfb,993b861a1d5d5955
Normal-6747.png,Normal,Normal/images/Normal-6747.png,Normal/masks/Normal-6747.png,True,G15992,train,15086874b796528dbf0d6432d3c9e3e2,9979841e5ec51ae9
Normal-6748.png,Normal,Normal/images/Normal-6748.png,Normal/masks/Normal-6748.png,True,G15993,train,7badc3c9fb4c83f3dc6d347b73053ced,817cc61a1c4fbe1d
Normal-6749.png,Normal,Normal/images/Normal-6749.png,Normal/masks/Normal-6749.png,True,G15994,train,fe2dcd60e6768d2724a1ddb2f9d685b8,dc3d42070b4a6f1f
Normal-675.png,Normal,Normal/images/Normal-675.png,Normal/masks/Normal-675.png,True,G15995,train,d8e747633a134a98cf629b77f4d33133,d02bc25ada597955
Normal-6750.png,Normal,Normal/images/Normal-6750.png,Normal/masks/Normal-6750.png,True,G15996,train,eb2c738a8001a85886f11cdfe75b2269,d928d71f4c1bb948
Normal-6751.png,Normal,Normal/images/Normal-6751.png,Normal/masks/Normal-6751.png,True,G15997,train,03c2776bacb28484e2a59f2145255f9f,d97f06390d58e5c2
Normal-6752.png,Normal,Normal/images/Normal-6752.png,Normal/masks/Normal-6752.png,True,G15998,train,494749f375773eccee08aaffb32284d5,817a86961a1f4f3b
Normal-6753.png,Normal,Normal/images/Normal-6753.png,Normal/masks/Normal-6753.png,True,G15999,train,c0ba9dc69a2f4a666c01eef9560b249f,dc3d42425d1f4e2d
Normal-6754.png,Normal,Normal/images/Normal-6754.png,Normal/masks/Normal-6754.png,True,G16000,train,6c0f1d8eb4769073b21a3699cd673988,d979c61a5f0d1a19
Normal-6755.png,Normal,Normal/images/Normal-6755.png,Normal/masks/Normal-6755.png,True,G16001,test,d077e5244fadb5a35aede8773a516020,dc2dd24b4f410b3d
Normal-6756.png,Normal,Normal/images/Normal-6756.png,Normal/masks/Normal-6756.png,True,G16002,val,0612dc1a5bd8050b0a10c0ffc6633bb0,806987571b574e1f
Normal-6757.png,Normal,Normal/images/Normal-6757.png,Normal/masks/Normal-6757.png,True,G16003,test,4bcd3326439cabfd6a32126c18748075,d92984165acd5e79
Normal-6758.png,Normal,Normal/images/Normal-6758.png,Normal/masks/Normal-6758.png,True,G16004,train,a99e794a83a6987e3f4454e88bb76106,d91d464a1d4a4f6d
Normal-6759.png,Normal,Normal/images/Normal-6759.png,Normal/masks/Normal-6759.png,True,G16005,test,8a9a6bcf17b6ee974c29fd0d577c0fba,893c460ff905fc55
Normal-676.png,Normal,Normal/images/Normal-676.png,Normal/masks/Normal-676.png,True,G16006,train,84f9e92d7149f010e55a199f9f685958,906dc31ae9693dc5
Normal-6760.png,Normal,Normal/images/Normal-6760.png,Normal/masks/Normal-6760.png,True,G16007,val,3e906cea0031f121de3d9c71efb9fe4e,dd2ec25b0a4e2d6c
Normal-6761.png,Normal,Normal/images/Normal-6761.png,Normal/masks/Normal-6761.png,True,G16008,test,f54379ca9837340243c7b8f03d5cbb60,89f8851ad7875a17
Normal-6762.png,Normal,Normal/images/Normal-6762.png,Normal/masks/Normal-6762.png,True,G16009,train,18c5c291d524246ae614b468d2cddc86,c42d434b098dedfc
Normal-6763.png,Normal,Normal/images/Normal-6763.png,Normal/masks/Normal-6763.png,True,G16010,val,b53dde8672c5adfee4ce0ebe0e30bfb9,887907165a4d7fb8
Normal-6764.png,Normal,Normal/images/Normal-6764.png,Normal/masks/Normal-6764.png,True,G16011,train,d4232a2a781beb44d5a1372210ba5d4f,887f431a5d3919ab
Normal-6765.png,Normal,Normal/images/Normal-6765.png,Normal/masks/Normal-6765.png,True,G16012,val,b15109926229733b833fa950ce101e27,997f061a0d1ab9ec
Normal-6766.png,Normal,Normal/images/Normal-6766.png,Normal/masks/Normal-6766.png,True,G16013,train,515cb35a53fac0a4bc74b3e66fb870c0,dd2fc29543c60fc1
Normal-6767.png,Normal,Normal/images/Normal-6767.png,Normal/masks/Normal-6767.png,True,G16014,train,f7bef1587fe4febec588ba94a91ada94,9169061e5a5f7879
Normal-6768.png,Normal,Normal/images/Normal-6768.png,Normal/masks/Normal-6768.png,True,G16015,train,6f47af2baedab703cf9a9fa224aff75e,dd2cd3430f4c6a1d
Normal-6769.png,Normal,Normal/images/Normal-6769.png,Normal/masks/Normal-6769.png,True,G16016,train,d6def8a6796a7ea2455b07e5294b563a,d57f060a0f4a1b4f
Normal-677.png,Normal,Normal/images/Normal-677.png,Normal/masks/Normal-677.png,True,G16017,train,2515a2fc393399462a887341fa4cfad6,802bc65a5a797979
Normal-6770.png,Normal,Normal/images/Normal-6770.png,Normal/masks/Normal-6770.png,True,G16018,train,f96daa44a874ef84e801e58616b33c82,cd2e505b454f0b6d
Normal-6771.png,Normal,Normal/images/Normal-6771.png,Normal/masks/Normal-6771.png,True,G16019,train,d8a1f1269aa6d3cd583b910c4ead39e7,dd2cc30b1d4c1d9d
Normal-6772.png,Normal,Normal/images/Normal-6772.png,Normal/masks/Normal-6772.png,True,G16020,train,a3d1e7c3a737339855ae76af6e86d522,885f971a5968bab8
Normal-6773.png,Normal,Normal/images/Normal-6773.png,Normal/masks/Normal-6773.png,True,G16021,test,aa0b15480c735ecab928c6c251e9a252,cd3d420f1a4d3e5a
Normal-6774.png,Normal,Normal/images/Normal-6774.png,Normal/masks/Normal-6774.png,True,G16022,train,1ec6c1e837d9acf9bc64493ec0cfba84,813e07191f5e795c
Normal-6775.png,Normal,Normal/images/Normal-6775.png,Normal/masks/Normal-6775.png,True,G16023,train,57a69b1d90e189c28d9bc2dace05d8ca,9178865e18de5a79
Normal-6776.png,Normal,Normal/images/Normal-6776.png,Normal/masks/Normal-6776.png,True,G16024,train,5bb774cbee8ebd19b4bc138e3136c4d1,9939c61e11475ef8
Normal-6777.png,Normal,Normal/images/Normal-6777.png,Normal/masks/Normal-6777.png,True,G16025,train,ead7adc7681195134f82bc578bf4a4c0,9d3f464b186ce839
Normal-6778.png,Normal,Normal/images/Normal-6778.png,Normal/masks/Normal-6778.png,True,G16026,val,d9fd7c073f5597b9e8164436948d7f8b,dd3f2b6de4651202
Normal-6779.png,Normal,Normal/images/Normal-6779.png,Normal/masks/Normal-6779.png,True,G16027,val,e2315efaa6cc96cbc1f3d1f46c127cc6,d879c716144f5f18
Normal-678.png,Normal,Normal/images/Normal-678.png,Normal/masks/Normal-678.png,True,G16028,test,2819119629cf40d348582de98f3b0eb5,d046c71fcd6c4cc6
Normal-6780.png,Normal,Normal/images/Normal-6780.png,Normal/masks/Normal-6780.png,True,G16029,test,9837baca528ac02866db871c8e960c0f,d929861f1d0f3939
Normal-6781.png,Normal,Normal/images/Normal-6781.png,Normal/masks/Normal-6781.png,True,G16030,train,59dc31afc432d49e94131aa2a5c45a43,dc2ac30b0d4d3d3d
Normal-6782.png,Normal,Normal/images/Normal-6782.png,Normal/masks/Normal-6782.png,True,G16031,train,6651b5383a4faf58484c50e3ebc4794a,cc3cc34b4a4f2dcc
Normal-6783.png,Normal,Normal/images/Normal-6783.png,Normal/masks/Normal-6783.png,True,G16032,train,1333a05ceec2893beffbe9c68ec1a9ac,dd2ec61f584e3942
Normal-6784.png,Normal,Normal/images/Normal-6784.png,Normal/masks/Normal-6784.png,True,G16033,train,774b6016cdc57cd3290603ceec4f86ad,817b86149f4f185f
Normal-6785.png,Normal,Normal/images/Normal-6785.png,Normal/masks/Normal-6785.png,True,G16034,val,0316fadbd1a2ec4444153e7b9a08e3da,817b16185b3da8bd
Normal-6786.png,Normal,Normal/images/Normal-6786.png,Normal/masks/Normal-6786.png,True,G16035,train,e68241588ebd199a7e187fc69f601f85,997b46061f1f461e
Normal-6787.png,Normal,Normal/images/Normal-6787.png,Normal/masks/Normal-6787.png,True,G16036,train,ec58731717ca973cae86f5cbd681c147,917f0642195d4f79
Normal-6788.png,Normal,Normal/images/Normal-6788.png,Normal/masks/Normal-6788.png,True,G16037,train,3aef51db83c78adf713b66f6fff16f20,8938461f1c6f39b9
Normal-6789.png,Normal,Normal/images/Normal-6789.png,Normal/masks/Normal-6789.png,True,G16038,train,08db7d0d80aa64f007e3e331412d1f41,990d064a1d4d7d7d
Normal-679.png,Normal,Normal/images/Normal-679.png,Normal/masks/Normal-679.png,True,G16039,train,4208d07325d64f6610dac006c5f12051,91f8961a919db197
Normal-6790.png,Normal,Normal/images/Normal-6790.png,Normal/masks/Normal-6790.png,True,G16040,train,eeda9336e07c66407facebda66bfa88e,9969c616b91ee81b
Normal-6791.png,Normal,Normal/images/Normal-6791.png,Normal/masks/Normal-6791.png,True,G16041,train,4118d42f071e254b273352322901f36a,99798616554e1d5b
Normal-6792.png,Normal,Normal/images/Normal-6792.png,Normal/masks/Normal-6792.png,True,G16042,train,c574eb179a6b8b22ad3f065c6c10b191,8979071e18cd5abb
Normal-6793.png,Normal,Normal/images/Normal-6793.png,Normal/masks/Normal-6793.png,True,G16043,train,c6c7549947e38a6f7c684b9de6e5396b,887d871e9e471e19
Normal-6794.png,Normal,Normal/images/Normal-6794.png,Normal/masks/Normal-6794.png,True,G16044,train,21ad734f1be96d197f24de286d9b3d30,81390e16195ffe3a
Normal-6795.png,Normal,Normal/images/Normal-6795.png,Normal/masks/Normal-6795.png,True,G16045,train,79567da4455fc91c80d4e8c4ccb1fb79,dd2cd04353474b1f
Normal-6796.png,Normal,Normal/images/Normal-6796.png,Normal/masks/Normal-6796.png,True,G16046,val,67cbf108570d761a77595fdb0c9e555d,cc2d02c25f574b4f
Normal-6797.png,Normal,Normal/images/Normal-6797.png,Normal/masks/Normal-6797.png,True,G16047,val,5c501ce432dcb9bf87ceab1590d95177,893f96085e5b1c4f
Normal-6798.png,Normal,Normal/images/Normal-6798.png,Normal/masks/Normal-6798.png,True,G16048,test,30d0d23c55e26917390f0312e353dc9d,9865c61a551e5d4f
Normal-6799.png,Normal,Normal/images/Normal-6799.png,Normal/masks/Normal-6799.png,True,G16049,test,0f2eea6b73c5fe144a82ae83e36471fd,d93b84421f1d4b1f
Normal-68.png,Normal,Normal/images/Normal-68.png,Normal/masks/Normal-68.png,True,G16050,test,f1519f7c2da31c71ad9bc0a1c49f3b60,917fc63a8a4b38a5
Normal-680.png,Normal,Normal/images/Normal-680.png,Normal/masks/Normal-680.png,True,G16051,train,3a2aaa0b3c6021a20e3438c757807006,806acb5a5b5d38da
Normal-6800.png,Normal,Normal/images/Normal-6800.png,Normal/masks/Normal-6800.png,True,G16052,train,513c6e92021a4b57e427f65bcc8110ae,8979461e186f3ab9
Normal-6801.png,Normal,Normal/images/Normal-6801.png,Normal/masks/Normal-6801.png,True,G16053,train,c760dc6ba2c6655e89434164991d4a86,996f461a4e58796c
Normal-6802.png,Normal,Normal/images/Normal-6802.png,Normal/masks/Normal-6802.png,True,G16054,train,19a140084b4a40d19759606b9f610fc4,d81f821b3ce1adb4
Normal-6803.png,Normal,Normal/images/Normal-6803.png,Normal/masks/Normal-6803.png,True,G16055,train,8af33d6fc134cba9aa41b4a643cb502d,dc2dc2431f4f491d
Normal-6804.png,Normal,Normal/images/Normal-6804.png,Normal/masks/Normal-6804.png,True,G16056,train,8f7d159392752d1e1a39cc5296f5feb5,8868501f1b5d1f5f
Normal-6805.png,Normal,Normal/images/Normal-6805.png,Normal/masks/Normal-6805.png,True,G16057,train,652772653184a56c8c8620b3b98ab740,dd2c521b424b3e6b
Normal-6806.png,Normal,Normal/images/Normal-6806.png,Normal/masks/Normal-6806.png,True,G16058,train,c3e885b44e1fa9d8c9f85ddab18a2a3d,cc3dc31b4bc20f1d
Normal-6807.png,Normal,Normal/images/Normal-6807.png,Normal/masks/Normal-6807.png,True,G16059,train,12be5a61b9d22783453359ae5730da6a,817bd6161a4d2f39
Normal-6808.png,Normal,Normal/images/Normal-6808.png,Normal/masks/Normal-6808.png,True,G16060,train,95c07c75e8965411533b5ec7fbb37395,9929061f1e4e39fa
Normal-6809.png,Normal,Normal/images/Normal-6809.png,Normal/masks/Normal-6809.png,True,G16061,train,e9d3b2748b955737274269c3b2c0d9d0,817857c25e5d585b
Normal-681.png,Normal,Normal/images/Normal-681.png,Normal/masks/Normal-681.png,True,G16062,train,20053cfda6b34bb78ec4ffc967acbf34,906aca1add5b18db
Normal-6810.png,Normal,Normal/images/Normal-6810.png,Normal/masks/Normal-6810.png,True,G16063,train,b3e704903cb4bf0cadc28c0310b31a4b,9d2c025acbee68f8
Normal-6811.png,Normal,Normal/images/Normal-6811.png,Normal/masks/Normal-6811.png,True,G16064,val,b910ec70c57df06d73d90709562003d2,9c0c4b8b1decb4bc
Normal-6812.png,Normal,Normal/images/Normal-6812.png,Normal/masks/Normal-6812.png,True,G16065,val,6819618b634e57e3f5d52bf5e122b6f6,d42e434b4d4e4e6d
Normal-6813.png,Normal,Normal/images/Normal-6813.png,Normal/masks/Normal-6813.png,True,G16066,test,cf771c93b069cdb63151aa1f9d1fb2c9,91781e1b393ab3b2
Normal-6814.png,Normal,Normal/images/Normal-6814.png,Normal/masks/Normal-6814.png,True,G16067,val,ef7846e9e7102ae36f28ea9e16386da9,d43c8243591f4f2f
Normal-6815.png,Normal,Normal/images/Normal-6815.png,Normal/masks/Normal-6815.png,True,G16068,test,cf421b404de1393d8720814aa78a0444,9939c61a1b4f1979
Normal-6816.png,Normal,Normal/images/Normal-6816.png,Normal/masks/Normal-6816.png,True,G16069,train,c9a203f2c46055d774cd885e71227bad,dd3f421f0e203d2d
Normal-6817.png,Normal,Normal/images/Normal-6817.png,Normal/masks/Normal-6817.png,True,G16070,test,4039e338203aa3ead1c95a6d45988262,d4bcc359c349d14d
Normal-6818.png,Normal,Normal/images/Normal-6818.png,Normal/masks/Normal-6818.png,True,G16071,train,6ca3fc9aebeabfe142438cec59bd122a,c83d434b0f4b2f2e
Normal-6819.png,Normal,Normal/images/Normal-6819.png,Normal/masks/Normal-6819.png,True,G16072,train,335067426cdbbffaea0449ca0e7794a2,cc3d83031d4a7d1f
Normal-682.png,Normal,Normal/images/Normal-682.png,Normal/masks/Normal-682.png,True,G16073,train,5741baf9d795297b7c4af7bf9b6a2e42,812bc64acc4b7dd3
Normal-6820.png,Normal,Normal/images/Normal-6820.png,Normal/masks/Normal-6820.png,True,G16074,train,428c06192a0686490e423fbb4d830efe,913b061b79ecb8e1
Normal-6821.png,Normal,Normal/images/Normal-6821.png,Normal/masks/Normal-6821.png,True,G16075,train,7974650ffbf7a618a7ff0d0b228c256e,99fbc61e960e104f
Normal-6822.png,Normal,Normal/images/Normal-6822.png,Normal/masks/Normal-6822.png,True,G16076,train,95019992d406f1d9f1fd61529db6219e,ac07831b0d0f4f6f
Normal-6823.png,Normal,Normal/images/Normal-6823.png,Normal/masks/Normal-6823.png,True,G16077,val,d8a882727c0cab00e5868758da0d7516,89780e5ab94fa939
Normal-6824.png,Normal,Normal/images/Normal-6824.png,Normal/masks/Normal-6824.png,True,G16078,test,f308010592438ff0f2992e739cc8e271,d03d42e35f15494f
Normal-6825.png,Normal,Normal/images/Normal-6825.png,Normal/masks/Normal-6825.png,True,G16079,train,83f614d14b268d24ca7ee946897d357d,9978871a1ac75ab9
Normal-6826.png,Normal,Normal/images/Normal-6826.png,Normal/masks/Normal-6826.png,True,G16080,train,d2933a606da09291d3c374e5e18c582a,d92cc65b1dc41b1d
Normal-6827.png,Normal,Normal/images/Normal-6827.png,Normal/masks/Normal-6827.png,True,G16081,train,fc1ee0eb37f17858317621b7b2ee8de4,887945d6164f5f1a
Normal-6828.png,Normal,Normal/images/Normal-6828.png,Normal/masks/Normal-6828.png,True,G16082,train,e04fcf55ce0c96fdbee779b191ba6f01,d91d061e595959e5
Normal-6829.png,Normal,Normal/images/Normal-6829.png,Normal/masks/Normal-6829.png,True,G16083,train,765a4492e7e8ebdffa9ba53a76a1f79b,9979965a19077a39
Normal-683.png,Normal,Normal/images/Normal-683.png,Normal/masks/Normal-683.png,True,G16084,train,cdf6b62c23148696e10af217db670cc2,91ecc61acf6d08cd
Normal-6830.png,Normal,Normal/images/Normal-6830.png,Normal/masks/Normal-6830.png,True,G16085,train,e581cc1adaa8189b39550b0ac149d614,d92a541b1d4e1b3d
Normal-6831.png,Normal,Normal/images/Normal-6831.png,Normal/masks/Normal-6831.png,True,G16086,val,613f7bd4815ddf54a9c459f3e14365cf,d939d6431f0c4f0b
Normal-6832.png,Normal,Normal/images/Normal-6832.png,Normal/masks/Normal-6832.png,True,G16087,val,c8eace6da436bb1029cc882eefd236f5,c979c0575ad5465a
Normal-6833.png,Normal,Normal/images/Normal-6833.png,Normal/masks/Normal-6833.png,True,G16088,test,83f7d5abe7df74ea58b5cb129ca27e1f,d42dc35d4e4a2d2e
Normal-6834.png,Normal,Normal/images/Normal-6834.png,Normal/masks/Normal-6834.png,True,G16089,test,a5ae1191fb63ebdcbaf995a12fe1c5e2,dd2cd2d149d2cb19
Normal-6835.png,Normal,Normal/images/Normal-6835.png,Normal/masks/Normal-6835.png,True,G16090,train,2eb3c6f7ea3a50ac5ed2b1afceedd161,817956161a5e7e1b
Normal-6836.png,Normal,Normal/images/Normal-6836.png,Normal/masks/Normal-6836.png,True,G16091,train,dbcea162bc1d74fe269127d64e311f54,d93986161dce1b1d
Normal-6837.png,Normal,Normal/images/Normal-6837.png,Normal/masks/Normal-6837.png,True,G16092,train,5e9cd6fb566c6fefbdb1177579bad03d,8978871e184f7f38
Normal-6838.png,Normal,Normal/images/Normal-6838.png,Normal/masks/Normal-6838.png,True,G16093,train,1fdcc3ca383ddb8b4db9fc4ded03bd4a,dd2c535bc04f1c4d
Normal-6839.png,Normal,Normal/images/Normal-6839.png,Normal/masks/Normal-6839.png,True,G16094,train,677d83042535feb1e11e9a18648239df,8178161a7fe86975
Normal-684.png,Normal,Normal/images/Normal-684.png,Normal/masks/Normal-684.png,True,G16095,train,c56895af4b072d46cdb97ebdd4dd2219,d43cc21b9dcc3cb4
Normal-6840.png,Normal,Normal/images/Normal-6840.png,Normal/masks/Normal-6840.png,True,G16096,train,a9f0ff15f3f41aa03b7a33519f42cea0,c97b16423bc42fb1
Normal-6841.png,Normal,Normal/images/Normal-6841.png,Normal/masks/Normal-6841.png,True,G16097,train,492938a3979ac5613a58906d7661846d,d93d871a057e1947
Normal-6842.png,Normal,Normal/images/Normal-6842.png,Normal/masks/Normal-6842.png,True,G16098,train,e254cb66f2ce659d4575855ae9ca7777,c03c07434b1bdf2f
Normal-6843.png,Normal,Normal/images/Normal-6843.png,Normal/masks/Normal-6843.png,True,G16099,train,ab19b6715f31e76612b97d9c48ca3125,d92a560a1c613fb7
Normal-6844.png,Normal,Normal/images/Normal-6844.png,Normal/masks/Normal-6844.png,True,G16100,train,964b3a60d84eb2abbd5a40acf9f4b146,9d6c42431f4d0f3d
Normal-6845.png,Normal,Normal/images/Normal-6845.png,Normal/masks/Normal-6845.png,True,G16101,train,0a8ce86e9833e7471c558c9c0d9e64e5,897d061e194f1f1d
Normal-6846.png,Normal,Normal/images/Normal-6846.png,Normal/masks/Normal-6846.png,True,G16102,train,38c5c2b91231d55415b2b3de279d4564,dc4c4f0b0d4c4bde
Normal-6847.png,Normal,Normal/images/Normal-6847.png,Normal/masks/Normal-6847.png,True,G16103,train,09388ebfb0fb2925e2e889c2792f8be2,d92d861a194e7ab9
Normal-6848.png,Normal,Normal/images/Normal-6848.png,Normal/masks/Normal-6848.png,True,G16104,val,10434d8d173b42ee8351c00cf09b0009,9979061e1c5f5b1c
Normal-6849.png,Normal,Normal/images/Normal-6849.png,Normal/masks/Normal-6849.png,True,G16105,val,e3e5ebc2d78ba925466225d8262fb053,997d061a156a79e9
Normal-685.png,Normal,Normal/images/Normal-685.png,Normal/masks/Normal-685.png,True,G16106,test,9070b826bdc60e33b0017a6904d52821,813fc61a9cc473bc
Normal-6850.png,Normal,Normal/images/Normal-6850.png,Normal/masks/Normal-6850.png,True,G16107,test,eaa311f760ab392baf640ffdcd5b5b38,dc2c461b586c3cf5
Normal-6851.png,Normal,Normal/images/Normal-6851.png,Normal/masks/Normal-6851.png,True,G16108,val,f047bf0b02f3e24c682e0d236bc1239f,dd2a531f4c4c1d4e
Normal-6852.png,Normal,Normal/images/Normal-6852.png,Normal/masks/Normal-6852.png,True,G16109,test,895db3fc641ef739a81cc82a829d932a,997d841e4d5b194d
Normal-6853.png,Normal,Normal/images/Normal-6853.png,Normal/masks/Normal-6853.png,True,G16110,test,98fae62948cc53b3f3dfc32186f05435,dd6f16590d1805be
Normal-6854.png,Normal,Normal/images/Normal-6854.png,Normal/masks/Normal-6854.png,True,G16111,train,3b939126ae9a741b8d9cacf4d1496fec,dd2cd31b5c0a3d4c
Normal-6855.png,Normal,Normal/images/Normal-6855.png,Normal/masks/Normal-6855.png,True,G16112,train,3b88503b21dd72831c56347c4b732486,959b6ad84a6e8565
Normal-6856.png,Normal,Normal/images/Normal-6856.png,Normal/masks/Normal-6856.png,True,G16113,train,761683d2240092bda89d15967ccb6ae0,c97906861f1cef19
Normal-6857.png,Normal,Normal/images/Normal-6857.png,Normal/masks/Normal-6857.png,True,G16114,train,9c28904fc55bb8f97325f8ed21c229bc,d17f46125f42195b
Normal-6858.png,Normal,Normal/images/Normal-6858.png,Normal/masks/Normal-6858.png,True,G16115,train,9df7eaece2106a55f8d11ab9be3e44e2,d929861e1d4a3bbc
Normal-6859.png,Normal,Normal/images/Normal-6859.png,Normal/masks/Normal-6859.png,True,G16116,train,91718cbb5749271b83da22deff487a18,c93c47521bce5a39
Normal-686.png,Normal,Normal/images/Normal-686.png,Normal/masks/Normal-686.png,True,G16117,train,52db1e8bb5ef3274fc830edacc9c0178,986d871e835bb847
Normal-6860.png,Normal,Normal/images/Normal-6860.png,Normal/masks/Normal-6860.png,True,G16118,train,8f66f4c8c220ef487d564eee33852bc1,917bc61a5d41395d
Normal-6861.png,Normal,Normal/images/Normal-6861.png,Normal/masks/Normal-6861.png,True,G16119,train,92f0c05725c7dbb448d1f76ab5edb72b,817b06861a57df1e
Normal-6862.png,Normal,Normal/images/Normal-6862.png,Normal/masks/Normal-6862.png,True,G16120,train,64297e4e590245efac0dcc32fcd4b73e,cc2dc14b554f1d4d
Normal-6863.png,Normal,Normal/images/Normal-6863.png,Normal/masks/Normal-6863.png,True,G16121,train,92c5c7a2d54440271f4143037daed179,8129561e785e7d78
Normal-6864.png,Normal,Normal/images/Normal-6864.png,Normal/masks/Normal-6864.png,True,G16122,train,2666b7556b8fe1ded41c95fe635880ed,81791e06185fed7a
Normal-6865.png,Normal,Normal/images/Normal-6865.png,Normal/masks/Normal-6865.png,True,G16123,train,8dba7e7d1b5262f4934279b84fc8ec4d,9968961e151f5a1f
Normal-6866.png,Normal,Normal/images/Normal-6866.png,Normal/masks/Normal-6866.png,True,G16124,train,47ad4384e9bb5a034ff9688cf8890164,dc2c534b0f462f2d
Normal-6867.png,Normal,Normal/images/Normal-6867.png,Normal/masks/Normal-6867.png,True,G16125,train,fc7ede43e7d4e6dea6386e4ca982bc7b,997b861e188f3aaa
Normal-6868.png,Normal,Normal/images/Normal-6868.png,Normal/masks/Normal-6868.png,True,G16126,train,8ee7ab9749755807210616f2314f8cf1,d42f43571e4b1d1c
Normal-6869.png,Normal,Normal/images/Normal-6869.png,Normal/masks/Normal-6869.png,True,G16127,val,d2b06434d0048ecb61344ed7be614a72,d43d43410a8fed3d
Normal-687.png,Normal,Normal/images/Normal-687.png,Normal/masks/Normal-687.png,True,G16128,val,f1db2446b8f23b72122c0412d4752ea6,9139c67a786f3928
Normal-6870.png,Normal,Normal/images/Normal-6870.png,Normal/masks/Normal-6870.png,True,G16129,val,d0058aa959e98b8ff8aa442e8fc34791,d42ed3cb1dcb0a58
Normal-6871.png,Normal,Normal/images/Normal-6871.png,Normal/masks/Normal-6871.png,True,G16130,test,5bcaea34ac6768a484f885f01c177ef9,bd38c2929fc3829b
Normal-6872.png,Normal,Normal/images/Normal-6872.png,Normal/masks/Normal-6872.png,True,G16131,test,5d060da9b251c72b6c215c4195c691cf,dd2dd6821f55450e
Normal-6873.png,Normal,Normal/images/Normal-6873.png,Normal/masks/Normal-6873.png,True,G16132,train,5ea81c0f48ba6088393b97cab32284e1,897d061a055f7d65
Normal-6874.png,Normal,Normal/images/Normal-6874.png,Normal/masks/Normal-6874.png,True,G16133,train,f88e9d8c17a9f16b57915628ce81ef24,d42d434f0a2f0f1f
Normal-6875.png,Normal,Normal/images/Normal-6875.png,Normal/masks/Normal-6875.png,True,G16134,train,888955a10997965a49c8fd24ed51a5a6,dd2f471e1f49181c
Normal-6876.png,Normal,Normal/images/Normal-6876.png,Normal/masks/Normal-6876.png,True,G16135,train,a80f9c94cd62db0059a83926eeb59b7a,dd2e020b574f1e4d
Normal-6877.png,Normal,Normal/images/Normal-6877.png,Normal/masks/Normal-6877.png,True,G16136,train,937367bcc3c0e0bbb27c440065bc2ef6,dc3cc24b1bc30f1d
Normal-6878.png,Normal,Normal/images/Normal-6878.png,Normal/masks/Normal-6878.png,True,G16137,train,def56d9e752b996544dec4a5098fd4fc,d42e83190d5d1f4f
Normal-6879.png,Normal,Normal/images/Normal-6879.png,Normal/masks/Normal-6879.png,True,G16138,train,3d12d2d07adbf7a8e02707c8df483bef,c87d075e194d1e1d
Normal-688.png,Normal,Normal/images/Normal-688.png,Normal/masks/Normal-688.png,True,G16139,train,226fb69b69cd411cdd8f0754462ba035,94f4c75b8d2d6c0c
Normal-6880.png,Normal,Normal/images/Normal-6880.png,Normal/masks/Normal-6880.png,True,G16140,train,c7bf36edf42c6327aa339a91a01a5f95,dd2a530b4d450f1f
Normal-6881.png,Normal,Normal/images/Normal-6881.png,Normal/masks/Normal-6881.png,True,G16141,train,52c693402e3ed5eb23f109c12c8dc5f6,993cc61b1d4e1d5c
Normal-6882.png,Normal,Normal/images/Normal-6882.png,Normal/masks/Normal-6882.png,True,G16142,val,73079dc4a03307671a7d1168bebb9bf6,d43d434c0e2e3cfc
Normal-6883.png,Normal,Normal/images/Normal-6883.png,Normal/masks/Normal-6883.png,True,G16143,val,d2cd7c11d1b1ebe6a561d6e904b93412,d97946171a1f853a
Normal-6884.png,Normal,Normal/images/Normal-6884.png,Normal/masks/Normal-6884.png,True,G16144,test,9fc26c53dbad50c53a8bdbee07cf0a77,c13dc60e581f6e39
Normal-6885.png,Normal,Normal/images/Normal-6885.png,Normal/masks/Normal-6885.png,True,G16145,train,eae6010492cae3f965181595d0f79685,885d065e58e9a5ee
Normal-6886.png,Normal,Normal/images/Normal-6886.png,Normal/masks/Normal-6886.png,True,G16146,train,676a2c463c7f80a7bd55ae8203edc00c,996d861b851e594f
Normal-6887.png,Normal,Normal/images/Normal-6887.png,Normal/masks/Normal-6887.png,True,G16147,train,130aa341b0ddd9beece0583cdd2f4338,dc38530f483d3d2d
Normal-6888.png,Normal,Normal/images/Normal-6888.png,Normal/masks/Normal-6888.png,True,G16148,train,5dbf3a74a611d41215b37b4fa35a6ef8,882f075e596978f8
Normal-6889.png,Normal,Normal/images/Normal-6889.png,Normal/masks/Normal-6889.png,True,G16149,val,a107541ec8cd92e0a295afc86484b15b,dd3cd34b0f440a1f
Normal-689.png,Normal,Normal/images/Normal-689.png,Normal/masks/Normal-689.png,True,G16150,train,2eb38564931f73a2551edb22ef7ecaa8,987ad25e5d5918c5
Normal-6890.png,Normal,Normal/images/Normal-6890.png,Normal/masks/Normal-6890.png,True,G16151,test,6481ffee0f8cc374d40493d289b0252e,817bc69618565b79
Normal-6891.png,Normal,Normal/images/Normal-6891.png,Normal/masks/Normal-6891.png,True,G16152,test,c409d28b96ef030c02b3fe8dcadd2745,817a56843a957fb1
Normal-6892.png,Normal,Normal/images/Normal-6892.png,Normal/masks/Normal-6892.png,True,G16153,test,c86b1239b2bd52362862930c79762a48,992e0618577af06f
Normal-6893.png,Normal,Normal/images/Normal-6893.png,Normal/masks/Normal-6893.png,True,G16154,train,959a039e996cd3b072a5e87f6dba44b0,8179561f191f6d19
Normal-6894.png,Normal,Normal/images/Normal-6894.png,Normal/masks/Normal-6894.png,True,G16155,train,894d6cdc24108d7f5a99df94d679a2ca,99784717185f6b38
Normal-6895.png,Normal,Normal/images/Normal-6895.png,Normal/masks/Normal-6895.png,True,G16156,train,a200fe13d6167e6402b873e649b37254,807f161a5f1a1b5e
Normal-6896.png,Normal,Normal/images/Normal-6896.png,Normal/masks/Normal-6896.png,True,G16157,train,f2c2773655e760235b44cdf12ad7be8b,c87f521f5e0b1b0a
Normal-6897.png,Normal,Normal/images/Normal-6897.png,Normal/masks/Normal-6897.png,True,G16158,train,734910886492bc99844bc22d80528739,dc3dc2430eca4b3d
Normal-6898.png,Normal,Normal/images/Normal-6898.png,Normal/masks/Normal-6898.png,True,G16159,train,a0d25c93ac8510bc1d18e135053dfc42,d83d431b0fca1bca
Normal-6899.png,Normal,Normal/images/Normal-6899.png,Normal/masks/Normal-6899.png,True,G16160,train,565d1afc383e21f328ac889458c214de,9939d61b1d08b9b9
Normal-69.png,Normal,Normal/images/Normal-69.png,Normal/masks/Normal-69.png,True,G16161,train,9b54fb8fa70a5f106ea431ac02c30b3d,9579ce3a994839d8
Normal-690.png,Normal,Normal/images/Normal-690.png,Normal/masks/Normal-690.png,True,G16162,val,31bfb86c7c7604d0a36457a4ff7d1155,d47ec71b1d8d1c84
Normal-6900.png,Normal,Normal/images/Normal-6900.png,Normal/masks/Normal-6900.png,True,G16163,val,d3a909960edfa7e1cd2651b539a079f1,9c4e8259696cddc3
Normal-6901.png,Normal,Normal/images/Normal-6901.png,Normal/masks/Normal-6901.png,True,G16164,train,f27a9b4c6b6d1caca166ce29f3f5ff64,9979c61653471a5b
Normal-6902.png,Normal,Normal/images/Normal-6902.png,Normal/masks/Normal-6902.png,True,G16165,train,624522953699fa21a46b597e1c86d9f1,8978075a5e1c7a79
Normal-6903.png,Normal,Normal/images/Normal-6903.png,Normal/masks/Normal-6903.png,True,G16166,val,83f6823f1441d2e1171a441c122edb2c,d93dc2431b9c4f0b
Normal-6904.png,Normal,Normal/images/Normal-6904.png,Normal/masks/Normal-6904.png,True,G16167,train,f52da9338271326ca758cc82d3881739,897906165a59e97b
Normal-6905.png,Normal,Normal/images/Normal-6905.png,Normal/masks/Normal-6905.png,True,G16168,test,489102675a5071e2bb08648151faf5c7,883b571a1ad56bb8
Normal-6906.png,Normal,Normal/images/Normal-6906.png,Normal/masks/Normal-6906.png,True,G16169,test,9eb80f84b0c370442e324398b2ad7a39,996807561b4d7d78
Normal-6907.png,Normal,Normal/images/Normal-6907.png,Normal/masks/Normal-6907.png,True,G16170,train,02d39d77c1b53cdbbeb52e8163566d9c,dd28c61b5c4a39e5
Normal-6908.png,Normal,Normal/images/Normal-6908.png,Normal/masks/Normal-6908.png,True,G16171,train,2bcd549dba7e15e2adec28f56260c607,c969561f401f5d5a
Normal-6909.png,Normal,Normal/images/Normal-6909.png,Normal/masks/Normal-6909.png,True,G16172,train,d67eb67d21ffd15494c5f401e156f3d1,dd69871a451f5865
Normal-691.png,Normal,Normal/images/Normal-691.png,Normal/masks/Normal-691.png,True,G16173,train,dda6a1ca0878367c60253d3b11343b76,d928c64add491dcb
Normal-6910.png,Normal,Normal/images/Normal-6910.png,Normal/masks/Normal-6910.png,True,G16174,train,e86b813538e747967874c2b47a9c3463,d939c61e14471b5d
Normal-6911.png,Normal,Normal/images/Normal-6911.png,Normal/masks/Normal-6911.png,True,G16175,train,41ed8f5adbe7027e6cddbf5324096c1d,897907561a596d79
Normal-6912.png,Normal,Normal/images/Normal-6912.png,Normal/masks/Normal-6912.png,True,G16176,train,4acb4a9f827329f9922d87e19be13295,c93ec2191c69b9e6
Normal-6913.png,Normal,Normal/images/Normal-6913.png,Normal/masks/Normal-6913.png,True,G16177,train,0c95a9a9d3484e2402aeec516d794514,c93e06185d6df8e1
Normal-6914.png,Normal,Normal/images/Normal-6914.png,Normal/masks/Normal-6914.png,True,G16178,train,4780bac32c874d3e96698c3c253e0f58,883d041e1f1d5f5d
Normal-6915.png,Normal,Normal/images/Normal-6915.png,Normal/masks/Normal-6915.png,True,G16179,train,41f687f001189509bf3afec6d389db4c,817a861a5d5f3969
Normal-6916.png,Normal,Normal/images/Normal-6916.png,Normal/masks/Normal-6916.png,True,G16180,train,5692dd53c255180f53c537911dde6d5f,c969c6821b07cf5b
Normal-6917.png,Normal,Normal/images/Normal-6917.png,Normal/masks/Normal-6917.png,True,G16181,train,e7732424b9431007d69c64e5010c79dd,897d571a1845e93d
Normal-6918.png,Normal,Normal/images/Normal-6918.png,Normal/masks/Normal-6918.png,True,G16182,train,d0659fbb4cbc716a22741d778be2f3bc,9978c65a134f3a5a
Normal-6919.png,Normal,Normal/images/Normal-6919.png,Normal/masks/Normal-6919.png,True,G16183,train,27e41bd7f00cf6db39c9d61d6cd4ce98,d83d861b1d6538bc
Normal-692.png,Normal,Normal/images/Normal-692.png,Normal/masks/Normal-692.png,True,G16184,val,1eecbba1c7cc34f8a0f9ed35144969c6,d43ec33e2d383c64
Normal-6920.png,Normal,Normal/images/Normal-6920.png,Normal/masks/Normal-6920.png,True,G16185,val,b89071862792505a6bd5e88ecbff3d68,897987161ecf5e18
Normal-6921.png,Normal,Normal/images/Normal-6921.png,Normal/masks/Normal-6921.png,True,G16186,val,0caf21c7f81dc71978241adbe6f3103a,9979c61a194e3b39
Normal-6922.png,Normal,Normal/images/Normal-6922.png,Normal/masks/Normal-6922.png,True,G16187,test,4c64c7877f5b184e3c08f45ae23aa8a6,897f47561b194d19
Normal-6923.png,Normal,Normal/images/Normal-6923.png,Normal/masks/Normal-6923.png,True,G16188,test,a3e27c0832befd577c88ae317d790679,dd2ec61a1fc41a1d
Normal-6924.png,Normal,Normal/images/Normal-6924.png,Normal/masks/Normal-6924.png,True,G16189,train,84454d98418bfe42506e509c3eee463a,dc3d52c24b5dd243
Normal-6925.png,Normal,Normal/images/Normal-6925.png,Normal/masks/Normal-6925.png,True,G16190,test,99e700504ec3c330ecbf492283125067,d949471a1d4f1b39
Normal-6926.png,Normal,Normal/images/Normal-6926.png,Normal/masks/Normal-6926.png,True,G16191,train,11eacc71d74f616cfddb06634f61a474,dd2cc5131e4c5a5d
Normal-6927.png,Normal,Normal/images/Normal-6927.png,Normal/masks/Normal-6927.png,True,G16192,train,e1dd14c99dfa741681c059f55cdb38d7,dc2c425b4d4e39e5
Normal-6928.png,Normal,Normal/images/Normal-6928.png,Normal/masks/Normal-6928.png,True,G16193,train,e9c01210ad4d99abfd99301d143748ac,c06d96534b3d7c4a
Normal-6929.png,Normal,Normal/images/Normal-6929.png,Normal/masks/Normal-6929.png,True,G16194,train,17da3ca0f1799bb469039ad7ce67ffdc,907b05061e5e5e7e
Normal-693.png,Normal,Normal/images/Normal-693.png,Normal/masks/Normal-693.png,True,G16195,train,650e52fc0496a3435b8a57be7dcf559c,d1748638ad7d3c64
Normal-6930.png,Normal,Normal/images/Normal-6930.png,Normal/masks/Normal-6930.png,True,G16196,train,8117c5867964f9ce25023deb248680e9,d92dc61f184d1b39
Normal-6931.png,Normal,Normal/images/Normal-6931.png,Normal/masks/Normal-6931.png,True,G16197,train,ff06a5e4c197c3cbd9c59efd8702f063,d03d431b1d2e2d3d
Normal-6932.png,Normal,Normal/images/Normal-6932.png,Normal/masks/Normal-6932.png,True,G16198,train,1e079b5a9bb879ca320e0705f5588752,9c5c430b0e2f3ced
Normal-6933.png,Normal,Normal/images/Normal-6933.png,Normal/masks/Normal-6933.png,True,G16199,val,a57d4fd15cf1f736ae4ea56d271eb2b4,8978861a597d59d9
Normal-6934.png,Normal,Normal/images/Normal-6934.png,Normal/masks/Normal-6934.png,True,G16200,train,be15e8daace0a22e9b322cc590fbeca4,d9ead61dc419544e
Normal-6935.png,Normal,Normal/images/Normal-6935.png,Normal/masks/Normal-6935.png,True,G16201,train,7186c979cc8946267e13032beadff6e2,d179d605191e4b3b
Normal-6936.png,Normal,Normal/images/Normal-6936.png,Normal/masks/Normal-6936.png,True,G16202,train,c5f12bfc7420e1d08515688a62d5bcbb,cc3d430f086e3ff0
Normal-6937.png,Normal,Normal/images/Normal-6937.png,Normal/masks/Normal-6937.png,True,G16203,train,e57caa1b5a802b63a9c882131924493d,dc2e434b4c6e08fe
Normal-6938.png,Normal,Normal/images/Normal-6938.png,Normal/masks/Normal-6938.png,True,G16204,train,e5b68908d509507f6542be8055e03193,d02d4e1f5b6f180d
Normal-6939.png,Normal,Normal/images/Normal-6939.png,Normal/masks/Normal-6939.png,True,G16205,val,9e5318f1a0804c469dc445f0c7eaa3f8,897b07061f17465f
Normal-694.png,Normal,Normal/images/Normal-694.png,Normal/masks/Normal-694.png,True,G16206,val,180899d333271950bef48523fd9751ec,d06cc75bcdcd2982
Normal-6940.png,Normal,Normal/images/Normal-6940.png,Normal/masks/Normal-6940.png,True,G16207,train,6413a4e811695741a4bcf5e33a72f8f7,c42c4353194bb77e
Normal-6941.png,Normal,Normal/images/Normal-6941.png,Normal/masks/Normal-6941.png,True,G16208,test,a6ff0063fbeface5abc74400367213bc,c03c0fc11c2ef47f
Normal-6942.png,Normal,Normal/images/Normal-6942.png,Normal/masks/Normal-6942.png,True,G16209,test,a15c1e09577e713da26380e5ca70a816,993e86161ac55eda
Normal-6943.png,Normal,Normal/images/Normal-6943.png,Normal/masks/Normal-6943.png,True,G16210,test,36b8e075652b9197ac6a386c90398f0b,cc2c57435f0a4b4f
Normal-6944.png,Normal,Normal/images/Normal-6944.png,Normal/masks/Normal-6944.png,True,G16211,train,242a8d86070f42c45d9d2a5a6b1cee35,dd2ec24b1d486e9c
Normal-6945.png,Normal,Normal/images/Normal-6945.png,Normal/masks/Normal-6945.png,True,G16212,train,a6b6ff44268596f8b59f889a65339b22,dd3cc34859472f29
Normal-6946.png,Normal,Normal/images/Normal-6946.png,Normal/masks/Normal-6946.png,True,G16213,train,49e5c5e0f442c79d531a54d162e904e3,d92d861b19441ff9
Normal-6947.png,Normal,Normal/images/Normal-6947.png,Normal/masks/Normal-6947.png,True,G16214,train,548286371a69557d3a70243f2f737183,d42852d75f420f4f
Normal-6948.png,Normal,Normal/images/Normal-6948.png,Normal/masks/Normal-6948.png,True,G16215,test,2f0804a700545e1ce3786a7a3588d138,997b85061f841f1f
Normal-6949.png,Normal,Normal/images/Normal-6949.png,Normal/masks/Normal-6949.png,True,G16216,train,4771b6e84ea52b83f78090b2bfe4079f,917a06185b5f5a1f
Normal-695.png,Normal,Normal/images/Normal-695.png,Normal/masks/Normal-695.png,True,G16217,train,3a896b85c04fedf9665cea11fb8ba171,84ea871bed3c58c5
Normal-6950.png,Normal,Normal/images/Normal-6950.png,Normal/masks/Normal-6950.png,True,G16218,train,ed3a938a299cfd110ee1210fb2eeba74,887d87565a09575b
Normal-6951.png,Normal,Normal/images/Normal-6951.png,Normal/masks/Normal-6951.png,True,G16219,train,eb0ee2cce833eb3890ee40db121eb7fd,87796e38a186da59
Normal-6952.png,Normal,Normal/images/Normal-6952.png,Normal/masks/Normal-6952.png,True,G16220,train,3be40538d1f332c228771a36f28daae6,d03dc2c65a1bce4e
Normal-6953.png,Normal,Normal/images/Normal-6953.png,Normal/masks/Normal-6953.png,True,G16221,train,3fbe769b235a0d8e5ae5854c7a055f4a,dd294757844f9147
Normal-6954.png,Normal,Normal/images/Normal-6954.png,Normal/masks/Normal-6954.png,True,G16222,train,656a17e9ec910af3e89492bee3662b3e,9d7c460b486c6df1
Normal-6955.png,Normal,Normal/images/Normal-6955.png,Normal/masks/Normal-6955.png,True,G16223,train,34b440132b8109c8f5c303889a68f95a,8179971e184d7a79
Normal-6956.png,Normal,Normal/images/Normal-6956.png,Normal/masks/Normal-6956.png,True,G16224,train,0b12f085f92818e39087ecbbe0ba95c9,917806465f1aef6c
Normal-6957.png,Normal,Normal/images/Normal-6957.png,Normal/masks/Normal-6957.png,True,G16225,val,e1fa969fc87752b1317064a0d84ad61c,996dc61e784d7858
Normal-6958.png,Normal,Normal/images/Normal-6958.png,Normal/masks/Normal-6958.png,True,G16226,val,8dabc0c98238fbdb2feee022c3c1439f,d13fc618194f3959
Normal-6959.png,Normal,Normal/images/Normal-6959.png,Normal/masks/Normal-6959.png,True,G16227,val,3c3bf3e3de802c46d45242fa4acb190b,8929c65e19073e7b
Normal-696.png,Normal,Normal/images/Normal-696.png,Normal/masks/Normal-696.png,True,G16228,test,a4da5125fe2a1ba9b0de282a94f1b470,8129c21afc693ded
Normal-6960.png,Normal,Normal/images/Normal-6960.png,Normal/masks/Normal-6960.png,True,G16229,train,c6b2c5f2a8ad5b529a36e65642da03dc,817d841a5b6b7969
Normal-6961.png,Normal,Normal/images/Normal-6961.png,Normal/masks/Normal-6961.png,True,G16230,test,62ebdcdd5e8e8f9d2009a8b6c761832f,c82e461f4d2b38ed
Normal-6962.png,Normal,Normal/images/Normal-6962.png,Normal/masks/Normal-6962.png,True,G16231,train,bd39b5812d18807b7d6725e5143d5c0a,8d3c421b8d6cb9ec
Normal-6963.png,Normal,Normal/images/Normal-6963.png,Normal/masks/Normal-6963.png,True,G16232,train,0234a45ec9f016e3061167e2f4307dc7,916986521b7e3f38
Normal-6964.png,Normal,Normal/images/Normal-6964.png,Normal/masks/Normal-6964.png,True,G16233,train,cf03c03212f87e17b55170a4ad8bb05d,992b471a564e18ef
Normal-6965.png,Normal,Normal/images/Normal-6965.png,Normal/masks/Normal-6965.png,True,G16234,train,6104c0df9ab8da321901dbbb9b0baf98,876a1c9e5a3b583a
Normal-6966.png,Normal,Normal/images/Normal-6966.png,Normal/masks/Normal-6966.png,True,G16235,train,454f5f2722689d90218edcde87ebadea,8929d617f817a85b
Normal-6967.png,Normal,Normal/images/Normal-6967.png,Normal/masks/Normal-6967.png,True,G16236,train,6b7e31ce0b21c518297a4b3e6959a018,816904163fc77e97
Normal-6968.png,Normal,Normal/images/Normal-6968.png,Normal/masks/Normal-6968.png,True,G16237,train,2153b8a7b80bc86a9ee0b69d14f1e09b,c84d071e59e9a5ea
Normal-6969.png,Normal,Normal/images/Normal-6969.png,Normal/masks/Normal-6969.png,True,G16238,train,b2a93aa57b26921509fc63777c0b8b08,d43d42c30b4acf1f
Normal-697.png,Normal,Normal/images/Normal-697.png,Normal/masks/Normal-697.png,True,G16239,train,567aa165c63069c7855ca4c027e7770a,812bc25ad87939db
Normal-6970.png,Normal,Normal/images/Normal-6970.png,Normal/masks/Normal-6970.png,True,G16240,train,b14383c305af9f438b47384dd9531d34,cd3ec21d86499ae9
Normal-6971.png,Normal,Normal/images/Normal-6971.png,Normal/masks/Normal-6971.png,True,G16241,train,2be974615bade4389763523a391e0643,d97986de10c61f38
Normal-6972.png,Normal,Normal/images/Normal-6972.png,Normal/masks/Normal-6972.png,True,G16242,train,0f7eeeeed909d10add7179b9a4064ae1,897b475a190de939
Normal-6973.png,Normal,Normal/images/Normal-6973.png,Normal/masks/Normal-6973.png,True,G16243,val,ee35a6ed898dc2112768b671c34a3ce1,d93b57c65e095909
Normal-6974.png,Normal,Normal/images/Normal-6974.png,Normal/masks/Normal-6974.png,True,G16244,val,41da1cb7dd32a4a5c9da4398d60d2d39,d17bc6161a59691d
Normal-6975.png,Normal,Normal/images/Normal-6975.png,Normal/masks/Normal-6975.png,True,G16245,train,bc61d3ffe65e757ed66be145e54d5c51,d97ad653404f114f
Normal-6976.png,Normal,Normal/images/Normal-6976.png,Normal/masks/Normal-6976.png,True,G16246,test,dd4978a4b41ad5fbf30bc4f832705016,9979c61a1a8f3ab1
Normal-6977.png,Normal,Normal/images/Normal-6977.png,Normal/masks/Normal-6977.png,True,G16247,val,f12fc89a0928bdf6ab3a932ed4068ac6,dd3d06425d4fc86c
Normal-6978.png,Normal,Normal/images/Normal-6978.png,Normal/masks/Normal-6978.png,True,G16248,test,2d623c6939b04cd2a00416c6578185c2,b919c61e1d191d4f
Normal-6979.png,Normal,Normal/images/Normal-6979.png,Normal/masks/Normal-6979.png,True,G16249,test,1225c764d060b225b58c0c01c3ab55ab,897c061a7d8d79c5
Normal-698.png,Normal,Normal/images/Normal-698.png,Normal/masks/Normal-698.png,True,G16250,train,b4298af14a0df20ef0c2936519727f40,907dc73eaa6b3844
Normal-6980.png,Normal,Normal/images/Normal-6980.png,Normal/masks/Normal-6980.png,True,G16251,train,034028e3323a4bd7a0de5910382ac063,9129de06591edf0b
Normal-6981.png,Normal,Normal/images/Normal-6981.png,Normal/masks/Normal-6981.png,True,G16252,train,445bcfc607c58fedd618ecd62ae7e645,f91806861ff878f1
Normal-6982.png,Normal,Normal/images/Normal-6982.png,Normal/masks/Normal-6982.png,True,G16253,train,3265baa4028fcbf38644a0c25c14aa31,dd2c021bc51f1d6d
Normal-6983.png,Normal,Normal/images/Normal-6983.png,Normal/masks/Normal-6983.png,True,G16254,train,e4d04c71382813b06765b2e4dffb206b,dd2cc21b0b1a2fad
Normal-6984.png,Normal,Normal/images/Normal-6984.png,Normal/masks/Normal-6984.png,True,G16255,train,eb42c8871fe1a2cc9a0c7f785ef44750,9d2e020d0d3b4cff
Normal-6985.png,Normal,Normal/images/Normal-6985.png,Normal/masks/Normal-6985.png,True,G16256,train,bec897f62d81a932b6d2b4ab534a339c,912e061f071e5d5f
Normal-6986.png,Normal,Normal/images/Normal-6986.png,Normal/masks/Normal-6986.png,True,G16257,test,71bdf7bc19942bcbb4d9ef63c41b8a45,806c575b7f853aa4
Normal-6987.png,Normal,Normal/images/Normal-6987.png,Normal/masks/Normal-6987.png,True,G16258,train,6549b81b5d29d98449118d37b35b7c9e,887d86161a7d1d3b
Normal-6988.png,Normal,Normal/images/Normal-6988.png,Normal/masks/Normal-6988.png,True,G16259,train,1d47c55ca22d8b8cda5a7b2dd4599d46,91af870b051f3d0f
Normal-6989.png,Normal,Normal/images/Normal-6989.png,Normal/masks/Normal-6989.png,True,G16260,train,012ca7bd9c5049c795056cdd7d5690ac,dd2dc61b0c4e396c
Normal-699.png,Normal,Normal/images/Normal-699.png,Normal/masks/Normal-699.png,True,G16261,train,6ea89fed79a90ca160d511eb4d8c34f9,90bcc75f5249385e
Normal-6990.png,Normal,Normal/images/Normal-6990.png,Normal/masks/Normal-6990.png,True,G16262,val,e4ee949eaa7fd1f5cb55648a6c390fea,dd2cd2435b424f1d
Normal-6991.png,Normal,Normal/images/Normal-6991.png,Normal/masks/Normal-6991.png,True,G16263,test,c792e7bba43ab158aeb5c98cfdca604b,9979871a186d78b9
Normal-6992.png,Normal,Normal/images/Normal-6992.png,Normal/masks/Normal-6992.png,True,G16264,train,9cff21b6392d3c1fe813035c6bfe2551,996e85121fc55e78
Normal-6993.png,Normal,Normal/images/Normal-6993.png,Normal/masks/Normal-6993.png,True,G16265,train,06086d213b6d3aa45044d26975534055,99ef8618451f115f
Normal-6994.png,Normal,Normal/images/Normal-6994.png,Normal/masks/Normal-6994.png,True,G16266,train,d3650ea94a9f97078e7038ebe7a704bc,9938c6511b4e7f19
Normal-6995.png,Normal,Normal/images/Normal-6995.png,Normal/masks/Normal-6995.png,True,G16267,train,360f9853b35c1e67308826f17db54434,cd3fc6431d0c683d
Normal-6996.png,Normal,Normal/images/Normal-6996.png,Normal/masks/Normal-6996.png,True,G16268,val,6598b4011e55207fa864ea40b2d03f5c,cc3c424b8acf2fb8
Normal-6997.png,Normal,Normal/images/Normal-6997.png,Normal/masks/Normal-6997.png,True,G16269,val,1bba42b2054f6d18dc6e35d8d124ada3,9178061a5f5d68f9
Normal-6998.png,Normal,Normal/images/Normal-6998.png,Normal/masks/Normal-6998.png,True,G16270,test,c8fc7f63c3ca36f6f6c59de7ec127d0f,dd2ec25b4f454b0c
Normal-6999.png,Normal,Normal/images/Normal-6999.png,Normal/masks/Normal-6999.png,True,G16271,train,6ba4b1f9a43fb189bef24d856261c432,89a7961a0539cd7d
Normal-7.png,Normal,Normal/images/Normal-7.png,Normal/masks/Normal-7.png,True,G16272,train,900850d442ddd08283cf567ba0b02778,dd28d30f1f153839
Normal-70.png,Normal,Normal/images/Normal-70.png,Normal/masks/Normal-70.png,True,G16273,train,f5f0a8375f055ba4996ede84070b5f1f,907cc71b8d6d1cc6
Normal-700.png,Normal,Normal/images/Normal-700.png,Normal/masks/Normal-700.png,True,G16274,train,1f5408ad487fa4c9525530b650aaaa8c,916ad65a1c291d9f
Normal-7000.png,Normal,Normal/images/Normal-7000.png,Normal/masks/Normal-7000.png,True,G16275,train,8f3ff2d48f4b38ba807a78413f74cc5c,8178861e5b5b4e7a
Normal-7001.png,Normal,Normal/images/Normal-7001.png,Normal/masks/Normal-7001.png,True,G16276,train,ace5ed8285542425caba4cf7405e4d77,9d788616939e1e17
Normal-7002.png,Normal,Normal/images/Normal-7002.png,Normal/masks/Normal-7002.png,True,G16277,train,4272b6d5ac87cada8510b6ec23e86c37,893d865a18ce7e78
Normal-7003.png,Normal,Normal/images/Normal-7003.png,Normal/masks/Normal-7003.png,True,G16278,train,e8ef48d31806105d89ac4eec25257f81,dc3dc30f4e4a0e4e
Normal-7004.png,Normal,Normal/images/Normal-7004.png,Normal/masks/Normal-7004.png,True,G16279,train,fc321a5b6f7c3a94a5050a36a1b1ec5e,d12952175a0f793b
Normal-7005.png,Normal,Normal/images/Normal-7005.png,Normal/masks/Normal-7005.png,True,G16280,train,b9fdb1d6b57cc8a1020b758a7dc9a4ea,c42d430cafe6b4d3
Normal-7006.png,Normal,Normal/images/Normal-7006.png,Normal/masks/Normal-7006.png,True,G16281,train,e62dc63615b21acff943d328579a80f5,bc38530f550f1e4b
Normal-7007.png,Normal,Normal/images/Normal-7007.png,Normal/masks/Normal-7007.png,True,G16282,train,4557ffb82ea7273a483dfe20c7902e8f,890f861a5f5a595e
Normal-7008.png,Normal,Normal/images/Normal-7008.png,Normal/masks/Normal-7008.png,True,G16283,train,798596f91f3bddccc3217ed0aa512803,dc2d520f1e0e7d19
Normal-7009.png,Normal,Normal/images/Normal-7009.png,Normal/masks/Normal-7009.png,True,G16284,val,941c69cd1febcb5f91ae7d763455b604,817a57861a9d7a99
Normal-701.png,Normal,Normal/images/Normal-701.png,Normal/masks/Normal-701.png,True,G16285,test,cfc81a594aa64a77b551f157bf1b332a,916b061a597f79c1
Normal-7010.png,Normal,Normal/images/Normal-7010.png,Normal/masks/Normal-7010.png,True,G16286,train,0780658e35cea88f8b1757f385f80734,897b07121a5d7a9d
Normal-7011.png,Normal,Normal/images/Normal-7011.png,Normal/masks/Normal-7011.png,True,G16287,val,b919ae4e1ce2024d079a1e3d5036feeb,d42e47835f08fea2
Normal-7012.png,Normal,Normal/images/Normal-7012.png,Normal/masks/Normal-7012.png,True,G16288,val,a1820e8af0c5687925f3b0a00dbd9150,d13946471e1e4d9b
Normal-7013.png,Normal,Normal/images/Normal-7013.png,Normal/masks/Normal-7013.png,True,G16289,test,6093db1f56d0b7340a417350c1e17d3b,992f861a5ceab8e1
Normal-7014.png,Normal,Normal/images/Normal-7014.png,Normal/masks/Normal-7014.png,True,G16290,test,7a511ec4adbdc7ed778c9b080a132741,9979c7561e544e1c
Normal-7015.png,Normal,Normal/images/Normal-7015.png,Normal/masks/Normal-7015.png,True,G16291,train,c4e943d1605ef9976d16a325d15242fd,dc2d42d32b8a2f2d
Normal-7016.png,Normal,Normal/images/Normal-7016.png,Normal/masks/Normal-7016.png,True,G16292,train,264ca2f06ae141e98e7f215f87fbbd1d,dd2cc20f5c4f3c68
Normal-7017.png,Normal,Normal/images/Normal-7017.png,Normal/masks/Normal-7017.png,True,G16293,train,7fddd6cdd43ee04ca8265e04a4a48cd7,887dc34b0a4f0f4f
Normal-7018.png,Normal,Normal/images/Normal-7018.png,Normal/masks/Normal-7018.png,True,G16294,train,b2de91cbbdb8ade3d9d79375fb9a5788,837f101f7a1b3c38
Normal-7019.png,Normal,Normal/images/Normal-7019.png,Normal/masks/Normal-7019.png,True,G16295,train,c3870ae1421cf932ff6bc441aeef397d,992e861b59477ae8
Normal-702.png,Normal,Normal/images/Normal-702.png,Normal/masks/Normal-702.png,True,G16296,train,3f20f2b6e71847019ae7f7d8037d101b,d12fa71b0f414d55
Normal-7020.png,Normal,Normal/images/Normal-7020.png,Normal/masks/Normal-7020.png,True,G16297,train,4550e453b78a93a9012553582ca993d0,88b827875e9a5c4f
Normal-7021.png,Normal,Normal/images/Normal-7021.png,Normal/masks/Normal-7021.png,True,G16298,train,d1f6b017777521fb9cc294a4bdadd3d0,dd2cd24b1d4a0b3d
Normal-7022.png,Normal,Normal/images/Normal-7022.png,Normal/masks/Normal-7022.png,True,G16299,train,2ca2a4774cb42ab083002f395e729acb,dd6c071b1b05783d
Normal-7023.png,Normal,Normal/images/Normal-7023.png,Normal/masks/Normal-7023.png,True,G16300,train,f0d6de605d68695131d0c703982854a9,99780650195ff979
Normal-7024.png,Normal,Normal/images/Normal-7024.png,Normal/masks/Normal-7024.png,True,G16301,train,9f929aeeb27d5cd508c8eb7cd6090bc5,dc3d424b1b842f3d
Normal-7025.png,Normal,Normal/images/Normal-7025.png,Normal/masks/Normal-7025.png,True,G16302,train,904b78de315c37dfbbb8fd93d22342d2,c92cc71b1e461bbc
Normal-7026.png,Normal,Normal/images/Normal-7026.png,Normal/masks/Normal-7026.png,True,G16303,val,309f7a3c385aeeefdd2d40fe43801f26,fd0c161ac578fce0
Normal-7027.png,Normal,Normal/images/Normal-7027.png,Normal/masks/Normal-7027.png,True,G16304,train,4cf6e25b123e6e6782e772e832ef7913,d43c0bc3df0b904f
Normal-7028.png,Normal,Normal/images/Normal-7028.png,Normal/masks/Normal-7028.png,True,G16305,val,f390645a0dca939a6fdda4082abaffab,dd2cc75392471a5a
Normal-7029.png,Normal,Normal/images/Normal-7029.png,Normal/masks/Normal-7029.png,True,G16306,test,13824fed316e03217d0a0258ff6af529,9939061e1e4e78f9
Normal-703.png,Normal,Normal/images/Normal-703.png,Normal/masks/Normal-703.png,True,G16307,val,8a411cd3170ad0ad42a51f7cd7c4ca91,912bc61af84939f9
Normal-7030.png,Normal,Normal/images/Normal-7030.png,Normal/masks/Normal-7030.png,True,G16308,train,5f4f83dcbf754340bce63d7561543926,9c6ec31b4d4c1ccd
Normal-7031.png,Normal,Normal/images/Normal-7031.png,Normal/masks/Normal-7031.png,True,G16309,test,6ee5e8e81df3fed28824f89de1ed0771,d56f8612192f0d1f
Normal-7032.png,Normal,Normal/images/Normal-7032.png,Normal/masks/Normal-7032.png,True,G16310,test,067af712ad5834629ad5fd54bc90b162,8978161a5f185f79
Normal-7033.png,Normal,Normal/images/Normal-7033.png,Normal/masks/Normal-7033.png,True,G16311,train,d5b6eab65d517b47548f2fbd0c7dcabd,d96d851f5a4d1b48
Normal-7034.png,Normal,Normal/images/Normal-7034.png,Normal/masks/Normal-7034.png,True,G16312,train,943c825342cdc5cad7e08c1e42013a83,c93f4017174d1e5b
Normal-7035.png,Normal,Normal/images/Normal-7035.png,Normal/masks/Normal-7035.png,True,G16313,train,55273dfcad820d592e3db2fb9a07129d,9979855a190d5aed
Normal-7036.png,Normal,Normal/images/Normal-7036.png,Normal/masks/Normal-7036.png,True,G16314,train,6d1fc30637a3095129fdfec8af7fd0f4,817806161b5facfb
Normal-7037.png,Normal,Normal/images/Normal-7037.png,Normal/masks/Normal-7037.png,True,G16315,train,96966c40105c9f166d4cbc17c37b9198,dd2a571a5d013d59
Normal-7038.png,Normal,Normal/images/Normal-7038.png,Normal/masks/Normal-7038.png,True,G16316,val,3472fc57145f531742bbe406bea17823,9979c5049a17ed1e
Normal-7039.png,Normal,Normal/images/Normal-7039.png,Normal/masks/Normal-7039.png,True,G16317,train,05dece80241bdbd325df7cc8df06a805,8979861e595d5959
Normal-704.png,Normal,Normal/images/Normal-704.png,Normal/masks/Normal-704.png,True,G16318,train,7427bf7d691cbc62998cac0f674bdced,d5bd861b514f3c0c
Normal-7040.png,Normal,Normal/images/Normal-7040.png,Normal/masks/Normal-7040.png,True,G16319,train,b45f4dda1c0c7871a5590de72208976d,cd3dd24f4a820fac
Normal-7041.png,Normal,Normal/images/Normal-7041.png,Normal/masks/Normal-7041.png,True,G16320,train,da3dfe5c72ad75eac89cd3aec017d649,dd2ac21f424f194f
Normal-7042.png,Normal,Normal/images/Normal-7042.png,Normal/masks/Normal-7042.png,True,G16321,train,1364297fe8f266cbba87cc8f05dc2275,8879071e1bcd7ad1
Normal-7043.png,Normal,Normal/images/Normal-7043.png,Normal/masks/Normal-7043.png,True,G16322,train,20d387a9b189be83c07f137f46103f53,c83c4b0eb8f4d1da
Normal-7044.png,Normal,Normal/images/Normal-7044.png,Normal/masks/Normal-7044.png,True,G16323,train,dc8bce098216382d85cc6b5625879b08,d92c0e195d4cf8e6
Normal-7045.png,Normal,Normal/images/Normal-7045.png,Normal/masks/Normal-7045.png,True,G16324,train,8f92398db910a1d1f4a88b57a81ee144,807987161f1d1f1f
Normal-7046.png,Normal,Normal/images/Normal-7046.png,Normal/masks/Normal-7046.png,True,G16325,val,1e7f05f2605f65394dc00166e1b7a800,9969961a1b4f38b9
Normal-7047.png,Normal,Normal/images/Normal-7047.png,Normal/masks/Normal-7047.png,True,G16326,test,2482c38806117fa82db8b67fc4401a43,81681e1b59efe8a9
Normal-7048.png,Normal,Normal/images/Normal-7048.png,Normal/masks/Normal-7048.png,True,G16327,val,40edfd48da65a063f867e8fd60ffd4a1,81395616581ffd1b
Normal-7049.png,Normal,Normal/images/Normal-7049.png,Normal/masks/Normal-7049.png,True,G16328,test,0975954137093890ba54e9e3748b2776,d97fc61a5d491c0a
Normal-705.png,Normal,Normal/images/Normal-705.png,Normal/masks/Normal-705.png,True,G16329,test,b7a8e94657eda497a47a018e82733fc2,d47ec71e8d8568c4
Normal-7050.png,Normal,Normal/images/Normal-7050.png,Normal/masks/Normal-7050.png,True,G16330,val,41a223b8bb0e646421c3d6ebb6b8fe3f,fc2c53435b4b4d1a
Normal-7051.png,Normal,Normal/images/Normal-7051.png,Normal/masks/Normal-7051.png,True,G16331,test,2a0a7ce0499d5108634a8ab03ce65ff2,8938071e1f5e5f58
Normal-7052.png,Normal,Normal/images/Normal-7052.png,Normal/masks/Normal-7052.png,True,G16332,train,7d809576f8eb55ec342747d158da9bde,81785616185fed7a
Normal-7053.png,Normal,Normal/images/Normal-7053.png,Normal/masks/Normal-7053.png,True,G16333,train,a67b5933d53ba3d78a598db7f496878a,9178063f39687979
Normal-7054.png,Normal,Normal/images/Normal-7054.png,Normal/masks/Normal-7054.png,True,G16334,val,25d8ff9291ee153c3b33f19186043299,dd3d525341d24b59
Normal-7055.png,Normal,Normal/images/Normal-7055.png,Normal/masks/Normal-7055.png,True,G16335,val,70c85d85d629c6daddb40232ee3a2691,9179d61e1b8c3a39
Normal-7056.png,Normal,Normal/images/Normal-7056.png,Normal/masks/Normal-7056.png,True,G16336,train,6627796ffd195aec0c5b1cc5af0a2ab7,dc3e435f444f184d
Normal-7057.png,Normal,Normal/images/Normal-7057.png,Normal/masks/Normal-7057.png,True,G16337,train,78d4adc491ee7071c542791acff472e8,9d2807fe194d2d6c
Normal-7058.png,Normal,Normal/images/Normal-7058.png,Normal/masks/Normal-7058.png,True,G16338,train,cf8316ae01d64ecfe4d9b26cb0e69512,d02d42571f5dad0e
Normal-7059.png,Normal,Normal/images/Normal-7059.png,Normal/masks/Normal-7059.png,True,G16339,train,9f4622db4ea3d6c14076ea0ef75cadbd,d97f451a154a1b55
Normal-706.png,Normal,Normal/images/Normal-706.png,Normal/masks/Normal-706.png,True,G16340,train,55c10aa06dfdf0859df1e1feb236b179,d829d71ede4a1e48
Normal-7060.png,Normal,Normal/images/Normal-7060.png,Normal/masks/Normal-7060.png,True,G16341,train,a4075576600308f29b73e10e865cf9e6,d939c61716c61f15
Normal-7061.png,Normal,Normal/images/Normal-7061.png,Normal/masks/Normal-7061.png,True,G16342,train,25e4b1c3b91a3661e1efc2c005263f83,c92d861e165e3ada
Normal-7062.png,Normal,Normal/images/Normal-7062.png,Normal/masks/Normal-7062.png,True,G16343,train,f8cbcc3d0e71da3c42514065e6a76166,912dc61a19ce3f59
Normal-7063.png,Normal,Normal/images/Normal-7063.png,Normal/masks/Normal-7063.png,True,G16344,test,7f6f199505481e1cdf2d107b8a7bf8cd,817e061a5d587b5d
Normal-7064.png,Normal,Normal/images/Normal-7064.png,Normal/masks/Normal-7064.png,True,G16345,train,19727b601b80985353bebe5e0180db02,dd2dc40a4d0a3ded
Normal-7065.png,Normal,Normal/images/Normal-7065.png,Normal/masks/Normal-7065.png,True,G16346,train,16d4138f5440c31d8d92bbc4a358a128,91f946867e946e91
Normal-7066.png,Normal,Normal/images/Normal-7066.png,Normal/masks/Normal-7066.png,True,G16347,train,b78f5ea9d4f4fa81948138a454c33acd,9939c61ac92bbde0
Normal-7067.png,Normal,Normal/images/Normal-7067.png,Normal/masks/Normal-7067.png,True,G16348,test,7a925206e20606293891b800b8cd4993,dd2e430b0b4f0a6f
Normal-7068.png,Normal,Normal/images/Normal-7068.png,Normal/masks/Normal-7068.png,True,G16349,train,4e3dbb5f334619d6907dd05b6a387ab1,816917c75b1b5457
Normal-7069.png,Normal,Normal/images/Normal-7069.png,Normal/masks/Normal-7069.png,True,G16350,train,618d99467f71f517f0e617b553fd4f03,c17b578508951f9d
Normal-707.png,Normal,Normal/images/Normal-707.png,Normal/masks/Normal-707.png,True,G16351,train,5814432aa1bbad737feab15d9814a3c4,806dc73ae96d38c5
Normal-7070.png,Normal,Normal/images/Normal-7070.png,Normal/masks/Normal-7070.png,True,G16352,train,703ac5f3f00617f3345f6f4142020f03,c47cd3031b0b6d1f
Normal-7071.png,Normal,Normal/images/Normal-7071.png,Normal/masks/Normal-7071.png,True,G16353,train,e2c07a9728bedd2d2cc9363d7df6c08c,997807061e5f7ad8
Normal-7072.png,Normal,Normal/images/Normal-7072.png,Normal/masks/Normal-7072.png,True,G16354,train,dd2b041f6af2cbf8330863cb611d18a0,dd4f434b0d0c0baf
Normal-7073.png,Normal,Normal/images/Normal-7073.png,Normal/masks/Normal-7073.png,True,G16355,train,8556ad749d71ce865c9ffa7037d60f7b,fd2cc2031f4c1b1f
Normal-7074.png,Normal,Normal/images/Normal-7074.png,Normal/masks/Normal-7074.png,True,G16356,train,59ca33029d84fba57c6676b56fc460b0,986d865ad14e7aa9
Normal-7075.png,Normal,Normal/images/Normal-7075.png,Normal/masks/Normal-7075.png,True,G16357,train,7470f17cd7cafef84680bd0768fddac2,9d2c865b414f3acd
Normal-7076.png,Normal,Normal/images/Normal-7076.png,Normal/masks/Normal-7076.png,True,G16358,val,17a25c28db29f2a8836b12354dba7ea6,d96bd25f42594919
Normal-7077.png,Normal,Normal/images/Normal-7077.png,Normal/masks/Normal-7077.png,True,G16359,train,e9e4e67c344e442a2ac2463b4330e4ea,993987169cd71e1a
Normal-7078.png,Normal,Normal/images/Normal-7078.png,Normal/masks/Normal-7078.png,True,G16360,train,89cf9cd323724c5e48921fd3a9f7862a,980f821a4f3a6fea
Normal-7079.png,Normal,Normal/images/Normal-7079.png,Normal/masks/Normal-7079.png,True,G16361,val,73b5139f341eb5e895509b21bfaeb5d8,dc34f34b4bca490b
Normal-708.png,Normal,Normal/images/Normal-708.png,Normal/masks/Normal-708.png,True,G16362,train,275855af741207b8874374545569b94a,8d6e031b475a7d4c
Normal-7080.png,Normal,Normal/images/Normal-7080.png,Normal/masks/Normal-7080.png,True,G16363,val,6b2f495cf620d647010c45de564f4d56,995d165b59594c78
Normal-7081.png,Normal,Normal/images/Normal-7081.png,Normal/masks/Normal-7081.png,True,G16364,train,abd529e3be552ea09a607f8c8256fa2d,997f851e144f1e49
Normal-7082.png,Normal,Normal/images/Normal-7082.png,Normal/masks/Normal-7082.png,True,G16365,train,3dc30a09e94d5690058711fa6e673761,d97fc61ab909b890
Normal-7083.png,Normal,Normal/images/Normal-7083.png,Normal/masks/Normal-7083.png,True,G16366,test,9f17653d736b1d6569517b68a921f2f9,dc2ec34b0de2aaaa
Normal-7084.png,Normal,Normal/images/Normal-7084.png,Normal/masks/Normal-7084.png,True,G16367,train,0526066c058af0294b19ee9bee796268,81a97e97788d5178
Normal-7085.png,Normal,Normal/images/Normal-7085.png,Normal/masks/Normal-7085.png,True,G16368,test,cddfd586ad1b366f9d50d27c69b4039d,993906161b4f6eba
Normal-7086.png,Normal,Normal/images/Normal-7086.png,Normal/masks/Normal-7086.png,True,G16369,test,ff973953a2e5c9ed5f161fc7004854e3,d52e53094e2d1e6b
Normal-7087.png,Normal,Normal/images/Normal-7087.png,Normal/masks/Normal-7087.png,True,G16370,train,34c9e747df0797c196a14ad6e58cf3f5,d92d065eb9e8baa0
Normal-7088.png,Normal,Normal/images/Normal-7088.png,Normal/masks/Normal-7088.png,True,G16371,train,6add5953b4b63c91726e927ec54dec44,887d05c71b1f5e4a
Normal-7089.png,Normal,Normal/images/Normal-7089.png,Normal/masks/Normal-7089.png,True,G16372,test,3f554cb74d21598d1338b742cddeed11,dc254f1c4e2d4a6e
Normal-709.png,Normal,Normal/images/Normal-709.png,Normal/masks/Normal-709.png,True,G16373,train,001eaee19bc9388b780e2dc125b626c2,d938c71ee91839c9
Normal-7090.png,Normal,Normal/images/Normal-7090.png,Normal/masks/Normal-7090.png,True,G16374,train,d8027b7e56e45bd2d7ea3e0e4c6e4432,992fc41b07451bdd
Normal-7091.png,Normal,Normal/images/Normal-7091.png,Normal/masks/Normal-7091.png,True,G16375,test,0c243b7eaab433a67310ae06620bb44b,9969c41755471d5d
Normal-7092.png,Normal,Normal/images/Normal-7092.png,Normal/masks/Normal-7092.png,True,G16376,train,fc046070ca9931581f4df6270ef71796,8978c61a974fb869
Normal-7093.png,Normal,Normal/images/Normal-7093.png,Normal/masks/Normal-7093.png,True,G16377,train,c086f1cb0a8309c86aaca328f4316268,9121861e1b9f7aba
Normal-7094.png,Normal,Normal/images/Normal-7094.png,Normal/masks/Normal-7094.png,True,G16378,train,3b6b8d6d5656251f21607ad87c5bca8d,c03dc70b1dc70f1d
Normal-7095.png,Normal,Normal/images/Normal-7095.png,Normal/masks/Normal-7095.png,True,G16379,train,2d69551b735883429a53a1b913b5c8cd,d939020f3ae569b9
Normal-7096.png,Normal,Normal/images/Normal-7096.png,Normal/masks/Normal-7096.png,True,G16380,train,b36fc3fc58d21a1cb99b1f0e94382875,d959c71a1b491b39
Normal-7097.png,Normal,Normal/images/Normal-7097.png,Normal/masks/Normal-7097.png,True,G16381,train,a4b8303eaa91f4ac6051f3a2abe4e701,d979c55716c34b14
Normal-7098.png,Normal,Normal/images/Normal-7098.png,Normal/masks/Normal-7098.png,True,G16382,train,86ef4b2afdaef9656fe8dec30dca38ce,dd3cd04b0b434b1f
Normal-7099.png,Normal,Normal/images/Normal-7099.png,Normal/masks/Normal-7099.png,True,G16383,train,9de6323375cbd351f08ef65abb4b5bac,d42e43435f4e7c68
Normal-71.png,Normal,Normal/images/Normal-71.png,Normal/masks/Normal-71.png,True,G16384,train,8252f2362001d6f6414c87ff79467620,856ecf4b494569c3
Normal-710.png,Normal,Normal/images/Normal-710.png,Normal/masks/Normal-710.png,True,G16385,train,8d049687ffc08c7118ebb065bad56c16,916fc67a4d1d484d
Normal-7100.png,Normal,Normal/images/Normal-7100.png,Normal/masks/Normal-7100.png,True,G16386,val,2d76802e901d31e6a4739949cbdd2bdd,dd28d6421f1c6a3d
Normal-7101.png,Normal,Normal/images/Normal-7101.png,Normal/masks/Normal-7101.png,True,G16387,val,faf6185cd4eefa2a511e5c31cec3c051,8129d657555e5a1b
Normal-7102.png,Normal,Normal/images/Normal-7102.png,Normal/masks/Normal-7102.png,True,G16388,train,93a65db3f770d4973aa186da2704c4f3,8979c7861187571f
Normal-7103.png,Normal,Normal/images/Normal-7103.png,Normal/masks/Normal-7103.png,True,G16389,val,cf9d0e36370773527e8d9fc8712185fc,dc2ec34f0c4e2e3c
Normal-7104.png,Normal,Normal/images/Normal-7104.png,Normal/masks/Normal-7104.png,True,G16390,train,1694a8519beb798878bcfbfcc074a38e,d139c607181f4f3d
Normal-7105.png,Normal,Normal/images/Normal-7105.png,Normal/masks/Normal-7105.png,True,G16391,test,1c9d13d027ae86a9202b89fe2991817d,d42853430f5f4d1f
Normal-7106.png,Normal,Normal/images/Normal-7106.png,Normal/masks/Normal-7106.png,True,G16392,train,b81ea0ef3a7ccf0483c8bbf1377d7c38,d43c42c3cb0accef
Normal-7107.png,Normal,Normal/images/Normal-7107.png,Normal/masks/Normal-7107.png,True,G16393,train,c8291cc82e1cabd0c7381f18008d2d65,91694e061a39cd7f
Normal-7108.png,Normal,Normal/images/Normal-7108.png,Normal/masks/Normal-7108.png,True,G16394,train,4565051f3ef1386eaf3112f67e0037a6,dd2dc2d70fd0460d
Normal-7109.png,Normal,Normal/images/Normal-7109.png,Normal/masks/Normal-7109.png,True,G16395,train,96217682348334e06a49c10af35c0851,9168a406f81f7d3b
Normal-711.png,Normal,Normal/images/Normal-711.png,Normal/masks/Normal-711.png,True,G16396,train,2f2891d0621544dfb995c6f6662f0f70,916cea3a6d686c4e
Normal-7110.png,Normal,Normal/images/Normal-7110.png,Normal/masks/Normal-7110.png,True,G16397,train,1ee22528295c3b22f364f6e001bc9fb9,d02f861f1aac2eb9
Normal-7111.png,Normal,Normal/images/Normal-7111.png,Normal/masks/Normal-7111.png,True,G16398,train,7bb609822d436ad0af2c34459d7fba83,d43f830f0f0f0d0f
Normal-7112.png,Normal,Normal/images/Normal-7112.png,Normal/masks/Normal-7112.png,True,G16399,train,496e461e2af4b1ca662f3f8916ff950a,d17b06061a3d6779
Normal-7113.png,Normal,Normal/images/Normal-7113.png,Normal/masks/Normal-7113.png,True,G16400,train,41ee720d3f71dce24505eefca6d400ed,cd2c461b4f461a5f
Normal-7114.png,Normal,Normal/images/Normal-7114.png,Normal/masks/Normal-7114.png,True,G16401,train,94b1b932d4ede198ec9838a67ba2f9ed,997d871a945f1a8c
Normal-7115.png,Normal,Normal/images/Normal-7115.png,Normal/masks/Normal-7115.png,True,G16402,val,1853247727a0ad0b2dd0e6a31cbed7ac,dc2c42c74a2ceef4
Normal-7116.png,Normal,Normal/images/Normal-7116.png,Normal/masks/Normal-7116.png,True,G16403,train,425cfbd3255564553d05b3f86a366215,817ac5161a175f1f
Normal-7117.png,Normal,Normal/images/Normal-7117.png,Normal/masks/Normal-7117.png,True,G16404,train,ec0933c48512396eabb3c83f37416244,dd2cd34b0caca8bc
Normal-7118.png,Normal,Normal/images/Normal-7118.png,Normal/masks/Normal-7118.png,True,G16405,train,c2061adb0e8e4403780b068ce4bb51e1,9961565e1b5b7831
Normal-7119.png,Normal,Normal/images/Normal-7119.png,Normal/masks/Normal-7119.png,True,G16406,val,68877646f8e49ad449029d560e69b27d,dd2cc21f1e451f1c
Normal-712.png,Normal,Normal/images/Normal-712.png,Normal/masks/Normal-712.png,True,G16407,val,ff88a9428af97c98dda86204c07ca970,d06fc73a812d38b6
Normal-7120.png,Normal,Normal/images/Normal-7120.png,Normal/masks/Normal-7120.png,True,G16408,test,c14be37a17d8cd9fd0c024c4323c95e2,983d531b1d4d5d19
Normal-7121.png,Normal,Normal/images/Normal-7121.png,Normal/masks/Normal-7121.png,True,G16409,test,a55782019bb4ec1a3e794e6c1e8cc81f,d62d414b8f0b4e2f
Normal-7122.png,Normal,Normal/images/Normal-7122.png,Normal/masks/Normal-7122.png,True,G16410,test,cacd739d404e56009c08d0b1648ac52d,8179d557181ac65f
Normal-7123.png,Normal,Normal/images/Normal-7123.png,Normal/masks/Normal-7123.png,True,G16411,train,47fa6d4f530245d714bb6daed03275b2,907b47061a575f1e
Normal-7124.png,Normal,Normal/images/Normal-7124.png,Normal/masks/Normal-7124.png,True,G16412,train,6df170f65d53fa9431a11ab3b70531a2,c47c83439b4bae0f
Normal-7125.png,Normal,Normal/images/Normal-7125.png,Normal/masks/Normal-7125.png,True,G16413,train,9473ebd96932841778f613dda6bc71cb,d43d83151c0c8f7f
Normal-7126.png,Normal,Normal/images/Normal-7126.png,Normal/masks/Normal-7126.png,True,G16414,train,da7b3bdc37e96b7f4ea6e52d54fd46bb,91396a864a1b39ef
Normal-7127.png,Normal,Normal/images/Normal-7127.png,Normal/masks/Normal-7127.png,True,G16415,val,38baa783c732d16590f9993c0de803b3,dc3dc20f1fc20b1e
Normal-7128.png,Normal,Normal/images/Normal-7128.png,Normal/masks/Normal-7128.png,True,G16416,test,8124a24e6b100ea3eec149d7ba6af70d,9979861a174f1e59
Normal-7129.png,Normal,Normal/images/Normal-7129.png,Normal/masks/Normal-7129.png,True,G16417,test,8ee018d11c6a625e841e8e8431ac53b4,817f161a5969f9c1
Normal-713.png,Normal,Normal/images/Normal-713.png,Normal/masks/Normal-713.png,True,G16418,test,220f7987312789eaa37192299fbbaac4,91ab861abd49b5c6
Normal-7130.png,Normal,Normal/images/Normal-7130.png,Normal/masks/Normal-7130.png,True,G16419,train,24167f5eb176e5936e9afd4fb3ec3b7b,8178161a1f6f78b9
Normal-7131.png,Normal,Normal/images/Normal-7131.png,Normal/masks/Normal-7131.png,True,G16420,train,799bb1906513a3ca09d91734b8086273,dc2cd2435b430f5d
Normal-7132.png,Normal,Normal/images/Normal-7132.png,Normal/masks/Normal-7132.png,True,G16421,train,4922f268465329704ef32acefd2120ec,893d048693179fcf
Normal-7133.png,Normal,Normal/images/Normal-7133.png,Normal/masks/Normal-7133.png,True,G16422,train,7105f71ce3f3fc9223db2e3c53c6c81d,997f461a45591d47
Normal-7134.png,Normal,Normal/images/Normal-7134.png,Normal/masks/Normal-7134.png,True,G16423,val,4676aed2e8681417828c43dd93e4c237,d968861a1b4e7be8
Normal-7135.png,Normal,Normal/images/Normal-7135.png,Normal/masks/Normal-7135.png,True,G16424,train,a804c35eb13ca49718805e0e2714e587,f9571e0d21272727
Normal-7136.png,Normal,Normal/images/Normal-7136.png,Normal/masks/Normal-7136.png,True,G16425,val,4e33c86ed6e71f26221bd562e48ada2d,d42c52d35bc9474d
Normal-7137.png,Normal,Normal/images/Normal-7137.png,Normal/masks/Normal-7137.png,True,G16426,train,96921d7848705e5b44dbbf4b2478e66d,dc2e430b4ea638af
Normal-7138.png,Normal,Normal/images/Normal-7138.png,Normal/masks/Normal-7138.png,True,G16427,train,85219910d31a6f322076385eb6f98c20,e86987161a4d473f
Normal-7139.png,Normal,Normal/images/Normal-7139.png,Normal/masks/Normal-7139.png,True,G16428,train,5f649121503cb55588536784f148929d,817816461a1eef7b
Normal-714.png,Normal,Normal/images/Normal-714.png,Normal/masks/Normal-714.png,True,G16429,val,47821b671826d9fcfc4e1c9c602d1327,8129c67ad87939cd
Normal-7140.png,Normal,Normal/images/Normal-7140.png,Normal/masks/Normal-7140.png,True,G16430,train,29b637816ca08ca78cc0c8ce5edc0581,997b8716184f5e1a
Normal-7141.png,Normal,Normal/images/Normal-7141.png,Normal/masks/Normal-7141.png,True,G16431,train,5133df3b5d7e53f955657547b2806242,807ac71e1f4d3a17
Normal-7142.png,Normal,Normal/images/Normal-7142.png,Normal/masks/Normal-7142.png,True,G16432,train,4dfa44b19b02f0bdd04475349608c871,d52e865a0d2c7ea5
Normal-7143.png,Normal,Normal/images/Normal-7143.png,Normal/masks/Normal-7143.png,True,G16433,train,98668b2a7cb55327c2977d6c9f1a35b1,8179870698d7cfb4
Normal-7144.png,Normal,Normal/images/Normal-7144.png,Normal/masks/Normal-7144.png,True,G16434,train,2b369c0cb7e72984f5f1e03b0b5081a8,d929d7021f444f1f
Normal-7145.png,Normal,Normal/images/Normal-7145.png,Normal/masks/Normal-7145.png,True,G16435,train,a92c9b315d65fb78dccacc78c6eac52c,d02946567f484f7a
Normal-7146.png,Normal,Normal/images/Normal-7146.png,Normal/masks/Normal-7146.png,True,G16436,train,8bad13b66efff5c17681b6670f4a7014,dd2dd2431f550a1b
Normal-7147.png,Normal,Normal/images/Normal-7147.png,Normal/masks/Normal-7147.png,True,G16437,test,a522e5b10e6a597c361c98e158f7f584,993d065659c64f5a
Normal-7148.png,Normal,Normal/images/Normal-7148.png,Normal/masks/Normal-7148.png,True,G16438,test,a820f0c8ed7c6e920fa6ff759c36a9f1,a83ec30b1c6f791c
Normal-7149.png,Normal,Normal/images/Normal-7149.png,Normal/masks/Normal-7149.png,True,G16439,train,488bcd0eeddb26bac0b2db19fa0fdde5,d92d46171a457f1a
Normal-715.png,Normal,Normal/images/Normal-715.png,Normal/masks/Normal-715.png,True,G16440,train,6337d458f67969c6b3b0a54068e7adeb,943fc31f9cf01896
Normal-7150.png,Normal,Normal/images/Normal-7150.png,Normal/masks/Normal-7150.png,True,G16441,train,7fffe9bb2bc2371166aeb20daa2f3156,d93d021f1d446bf8
Normal-7151.png,Normal,Normal/images/Normal-7151.png,Normal/masks/Normal-7151.png,True,G16442,train,3c2e4c45754af23e295ff5a4de37eeea,d03d431f0a695f2d
Normal-7152.png,Normal,Normal/images/Normal-7152.png,Normal/masks/Normal-7152.png,True,G16443,train,1309966a88e0cae1cde832b0b919a513,897f461a5a453e1e
Normal-7153.png,Normal,Normal/images/Normal-7153.png,Normal/masks/Normal-7153.png,True,G16444,train,abd30b11a258f8726129be74cf0bdad3,997bc5461a16455f
Normal-7154.png,Normal,Normal/images/Normal-7154.png,Normal/masks/Normal-7154.png,True,G16445,val,0941cc6ae6d969a7f09db5057ba4cc41,997a45061e1f7b19
Normal-7155.png,Normal,Normal/images/Normal-7155.png,Normal/masks/Normal-7155.png,True,G16446,val,038f0d3cc120ed5ed950e72cd91b9f37,d93d42531acb0f1b
Normal-7156.png,Normal,Normal/images/Normal-7156.png,Normal/masks/Normal-7156.png,True,G16447,test,930a34f3bcc849b14c9efb4cec78f7cd,9369ded4585d4178
Normal-7157.png,Normal,Normal/images/Normal-7157.png,Normal/masks/Normal-7157.png,True,G16448,train,7e72c41fa56306f7aa846f2e171c24c2,d93d86171bc65a98
Normal-7158.png,Normal,Normal/images/Normal-7158.png,Normal/masks/Normal-7158.png,True,G16449,train,02a4f3cca921ac921123a04412f41218,d93b870e1d471b1c
Normal-7159.png,Normal,Normal/images/Normal-7159.png,Normal/masks/Normal-7159.png,True,G16450,train,db1e54d3ce627d9beb89b56eb75ccf67,9139c61e18575b79
Normal-716.png,Normal,Normal/images/Normal-716.png,Normal/masks/Normal-716.png,True,G16451,train,70da1135db2bc6761067189daf97d875,d0f6c739e5e420c6
Normal-7160.png,Normal,Normal/images/Normal-7160.png,Normal/masks/Normal-7160.png,True,G16452,train,8f8ec106a4535bc6d2235d755b22c6d9,dc29521f0f2d29bc
Normal-7161.png,Normal,Normal/images/Normal-7161.png,Normal/masks/Normal-7161.png,True,G16453,train,e54f0e47068ce6bf999441b2f5f55ca0,d87b45861e194b1f
Normal-7162.png,Normal,Normal/images/Normal-7162.png,Normal/masks/Normal-7162.png,True,G16454,train,01687da03945088968aa60725a0e88fe,dd2e50d30bc52b9c
Normal-7163.png,Normal,Normal/images/Normal-7163.png,Normal/masks/Normal-7163.png,True,G16455,train,33333ca51a8cf4d441756a99e6d10389,983b85161e0f5e5e
Normal-7164.png,Normal,Normal/images/Normal-7164.png,Normal/masks/Normal-7164.png,True,G16456,train,d1006a66dabce8f33e8e64ab18e301d1,d42d430d0a2f3ded
Normal-7165.png,Normal,Normal/images/Normal-7165.png,Normal/masks/Normal-7165.png,True,G16457,val,931b3c168b8cb66a5faf2afcb8eb675b,fd3d46c21b0c4f0b
Normal-7166.png,Normal,Normal/images/Normal-7166.png,Normal/masks/Normal-7166.png,True,G16458,val,1512dd1a2ad728c87484f7cb81bf56ee,d55f420a0e0f3d3d
Normal-7167.png,Normal,Normal/images/Normal-7167.png,Normal/masks/Normal-7167.png,True,G16459,test,cdae41c775f6fd94430ba6bc0849067b,9c4f861969ecd4c3
Normal-7168.png,Normal,Normal/images/Normal-7168.png,Normal/masks/Normal-7168.png,True,G16460,val,dc14c9ec133ef5c7090dfd1b5fb91c57,dd2ed20b1d464b1d
Normal-7169.png,Normal,Normal/images/Normal-7169.png,Normal/masks/Normal-7169.png,True,G16461,train,6ceba851abe1760ef568830bbf3223b4,8979c78616174f1e
Normal-717.png,Normal,Normal/images/Normal-717.png,Normal/masks/Normal-717.png,True,G16462,train,47e4d6a96dacb5c2229046422ffc7296,d02fc31bdc4b1cc6
Normal-7170.png,Normal,Normal/images/Normal-7170.png,Normal/masks/Normal-7170.png,True,G16463,train,de3d67543580f6ad1fb501aecca46c6c,dd3cc2c31bc44a3d
Normal-7171.png,Normal,Normal/images/Normal-7171.png,Normal/masks/Normal-7171.png,True,G16464,test,3d00f4b4b28c775ecc0f08399f8ed566,987dc61e184dad1e
Normal-7172.png,Normal,Normal/images/Normal-7172.png,Normal/masks/Normal-7172.png,True,G16465,test,2bfe31419fbd9b2b94e0e13cf57dc168,cc3c4b4b088bef3c
Normal-7173.png,Normal,Normal/images/Normal-7173.png,Normal/masks/Normal-7173.png,True,G16466,train,4e7393a3240579f55268401885940ff1,d96bc512d7451a1e
Normal-7174.png,Normal,Normal/images/Normal-7174.png,Normal/masks/Normal-7174.png,True,G16467,train,bdd4969fc45368944644d8e2a90b8e10,897f561a1d1f180f
Normal-7175.png,Normal,Normal/images/Normal-7175.png,Normal/masks/Normal-7175.png,True,G16468,test,90aab6a14f8b110d8b570e0c29485525,c13d4e961a39e569
Normal-7176.png,Normal,Normal/images/Normal-7176.png,Normal/masks/Normal-7176.png,True,G16469,train,a3ee300768ee6fb2fb95aeea03c8e2b9,dc2cc31b4d171e4d
Normal-7177.png,Normal,Normal/images/Normal-7177.png,Normal/masks/Normal-7177.png,True,G16470,train,f7d58983092eacefe15da15a4a4d519c,d42e42c34f4cec7c
Normal-7178.png,Normal,Normal/images/Normal-7178.png,Normal/masks/Normal-7178.png,True,G16471,train,c8778fa4e13c975992932d90a931840e,996fc61b865a1f42
Normal-7179.png,Normal,Normal/images/Normal-7179.png,Normal/masks/Normal-7179.png,True,G16472,train,62fcdc5061d226235be9bd2b4f6ce3ec,907d4f0b2fa43ce4
Normal-718.png,Normal,Normal/images/Normal-718.png,Normal/masks/Normal-718.png,True,G16473,train,dbd6368b54df0be274f14278dd155721,9165863aed78296d
Normal-7180.png,Normal,Normal/images/Normal-7180.png,Normal/masks/Normal-7180.png,True,G16474,train,157b6e11584f6ed17bec5dee2c5230f5,897b86961a195d5b
Normal-7181.png,Normal,Normal/images/Normal-7181.png,Normal/masks/Normal-7181.png,True,G16475,train,d7ca49fc82a919072880d7a1a14caf1f,d93cc4531bc44f1b
Normal-7182.png,Normal,Normal/images/Normal-7182.png,Normal/masks/Normal-7182.png,True,G16476,train,0eb5e82bf3af47cdcf5a56167d8c0f1e,c1344b4a1f681ffc
Normal-7183.png,Normal,Normal/images/Normal-7183.png,Normal/masks/Normal-7183.png,True,G16477,train,6bd67464af4c2d59dcb4a505e43075a0,8169561e19457f3d
Normal-7184.png,Normal,Normal/images/Normal-7184.png,Normal/masks/Normal-7184.png,True,G16478,train,8b2792590498d23be23bbd0d3e48057c,f96dc61a1847591d
Normal-7185.png,Normal,Normal/images/Normal-7185.png,Normal/masks/Normal-7185.png,True,G16479,train,b59a9b8a73989e89101b5ff8bc8d492d,997dc71a1e451a1d
Normal-7186.png,Normal,Normal/images/Normal-7186.png,Normal/masks/Normal-7186.png,True,G16480,train,a85e40a8906d75e4759e1b7f149e3f8c,8129ee87dc165753
Normal-7187.png,Normal,Normal/images/Normal-7187.png,Normal/masks/Normal-7187.png,True,G16481,train,929e5fadf4e6ca5fbbefda45d848b529,882dc7471f574e1c
Normal-7188.png,Normal,Normal/images/Normal-7188.png,Normal/masks/Normal-7188.png,True,G16482,train,cf4b8eb865228f05796a95e3b7a156d7,d43d43d10f2e2d1e
Normal-7189.png,Normal,Normal/images/Normal-7189.png,Normal/masks/Normal-7189.png,True,G16483,train,58682415ddfacbfd1785cc06c4c008f3,d93b461a19e97938
Normal-719.png,Normal,Normal/images/Normal-719.png,Normal/masks/Normal-719.png,True,G16484,val,607035b28e2991313fd5aa13badd8396,8139e63ae9793949
Normal-7190.png,Normal,Normal/images/Normal-7190.png,Normal/masks/Normal-7190.png,True,G16485,train,e727cab4170bafa058fa522d18414eca,d1398646181da77f
Normal-7191.png,Normal,Normal/images/Normal-7191.png,Normal/masks/Normal-7191.png,True,G16486,val,2abee56b71952e2e99b2e337dc287bbb,896886de1a4d3d79
Normal-7192.png,Normal,Normal/images/Normal-7192.png,Normal/masks/Normal-7192.png,True,G16487,val,bd73e03ff50623c58bfc82b4ffc7e6c6,d42e02d39f09c7c7
Normal-7193.png,Normal,Normal/images/Normal-7193.png,Normal/masks/Normal-7193.png,True,G16488,test,60f262496672d9eaaed36369deb7f71b,dc3cd34f444a2db4
Normal-7194.png,Normal,Normal/images/Normal-7194.png,Normal/masks/Normal-7194.png,True,G16489,test,0d9461c12358ef4cf31547ec07783a56,d52c071f0e1f5a6c
Normal-7195.png,Normal,Normal/images/Normal-7195.png,Normal/masks/Normal-7195.png,True,G16490,test,bf0010f7c4099d69a6f50a50e7fb96ea,9d2e470b0e0f7a6a
Normal-7196.png,Normal,Normal/images/Normal-7196.png,Normal/masks/Normal-7196.png,True,G16491,train,9e83c1f0a7d8e6b516f0f4865b0ac6ac,997f061a1e4e3b89
Normal-7197.png,Normal,Normal/images/Normal-7197.png,Normal/masks/Normal-7197.png,True,G16492,train,48d77cc1147f221168579cd4766edf05,d97f055616155b19
Normal-7198.png,Normal,Normal/images/Normal-7198.png,Normal/masks/Normal-7198.png,True,G16493,train,8bc29ac4f38ba990aa89c1a3fea8c7c4,887f078a1d696fb0
Normal-7199.png,Normal,Normal/images/Normal-7199.png,Normal/masks/Normal-7199.png,True,G16494,train,f0b6e7c5a3073b0396613bcbefc9b383,dd28d5c21bc54a9d
Normal-72.png,Normal,Normal/images/Normal-72.png,Normal/masks/Normal-72.png,True,G16495,train,3059ddbf13cddb6cc948d210527ec982,d0eec74bc9c169e0
Normal-720.png,Normal,Normal/images/Normal-720.png,Normal/masks/Normal-720.png,True,G16496,train,43ab26058b13d4e9c90d0c8d8723d0c4,81aac65adc4d5cd5
Normal-7200.png,Normal,Normal/images/Normal-7200.png,Normal/masks/Normal-7200.png,True,G16497,train,2acc3ef45191ff30b9304f9624eff1a9,91691e1a5b797958
Normal-7201.png,Normal,Normal/images/Normal-7201.png,Normal/masks/Normal-7201.png,True,G16498,train,002466a2e66a2a56721b6743a8f27e9f,996dc65b09cd3ab0
Normal-7202.png,Normal,Normal/images/Normal-7202.png,Normal/masks/Normal-7202.png,True,G16499,train,c7d22edf21c55b73ba1c444fc9e4ad88,890f065a5d79f8e2
Normal-7203.png,Normal,Normal/images/Normal-7203.png,Normal/masks/Normal-7203.png,True,G16500,train,fa3dc61f16bdf03c811a06b17e6e53b9,e92dc6c35b1d4549
Normal-7204.png,Normal,Normal/images/Normal-7204.png,Normal/masks/Normal-7204.png,True,G16501,val,1aa8d14aceef45602a705b4846ae7a2f,f92da45686d51a17
Normal-7205.png,Normal,Normal/images/Normal-7205.png,Normal/masks/Normal-7205.png,True,G16502,train,f35f0d0f14ede72456bcc225aab4a6aa,9968c61b1e4e3a1f
Normal-7206.png,Normal,Normal/images/Normal-7206.png,Normal/masks/Normal-7206.png,True,G16503,test,d70a33a6e8da56f109129809eb6eadcb,947f435b494d6c68
Normal-7207.png,Normal,Normal/images/Normal-7207.png,Normal/masks/Normal-7207.png,True,G16504,train,abfac52f1c72409fc1da7db743f8b1dc,89791618f9e939e1
Normal-7208.png,Normal,Normal/images/Normal-7208.png,Normal/masks/Normal-7208.png,True,G16505,train,4d7f3f9a5c2c9ff9126350552874a5d5,d92fc61a144f1b4d
Normal-7209.png,Normal,Normal/images/Normal-7209.png,Normal/masks/Normal-7209.png,True,G16506,val,d264d2c582dfaac6be9d529bacb950b8,9d68c60b571a5b4e
Normal-721.png,Normal,Normal/images/Normal-721.png,Normal/masks/Normal-721.png,True,G16507,train,d87808f6b8fbd29a8217140f81e20b45,9568ce1a99cf6985
Normal-7210.png,Normal,Normal/images/Normal-7210.png,Normal/masks/Normal-7210.png,True,G16508,train,d81419f3117b8732634c03d5dcf4dea7,d03d43033f6d6c3c
Normal-7211.png,Normal,Normal/images/Normal-7211.png,Normal/masks/Normal-7211.png,True,G16509,val,b51e3b229d45ffa6d27f4489c3e1c574,dc2e825309175f4f
Normal-7212.png,Normal,Normal/images/Normal-7212.png,Normal/masks/Normal-7212.png,True,G16510,test,5ebfa0b94709c5f9f8d59b93eb3703f1,98ed061b475a4d5e
Normal-7213.png,Normal,Normal/images/Normal-7213.png,Normal/masks/Normal-7213.png,True,G16511,test,31146a99df9508a8738bffd652bd8ff8,8179ce1e1b792a39
Normal-7214.png,Normal,Normal/images/Normal-7214.png,Normal/masks/Normal-7214.png,True,G16512,train,ceeced0fc7de18107bec9d17d7d5eb4e,887907961a19ef79
Normal-7215.png,Normal,Normal/images/Normal-7215.png,Normal/masks/Normal-7215.png,True,G16513,train,524bb6df4cbefacbcfa518e5c55b27d1,c17956861b1d4d4f
Normal-7216.png,Normal,Normal/images/Normal-7216.png,Normal/masks/Normal-7216.png,True,G16514,train,b1b6431c616667306d871c17bd62838e,cc3e43481e6b3d6c
Normal-7217.png,Normal,Normal/images/Normal-7217.png,Normal/masks/Normal-7217.png,True,G16515,train,49d467417f2c90fa9a0818c3bd4328fb,d97d865a1b0c4b39
Normal-7218.png,Normal,Normal/images/Normal-7218.png,Normal/masks/Normal-7218.png,True,G16516,train,d92373c12d1d948c6a032dbb2e1a6332,bc34c31f5c585d49
Normal-7219.png,Normal,Normal/images/Normal-7219.png,Normal/masks/Normal-7219.png,True,G16517,train,337719f0d63f29da3a53c2ae9636b079,c93cc651124a3ffa
Normal-722.png,Normal,Normal/images/Normal-722.png,Normal/masks/Normal-722.png,True,G16518,train,6a4e42d1d65def1b0d01df3fc2d155ce,d4f6e31aed4d14c0
Normal-7220.png,Normal,Normal/images/Normal-7220.png,Normal/masks/Normal-7220.png,True,G16519,train,1ea94d623a97d2c8f9e662ffa9054fc1,dd68d31b4b031b8b
Normal-7221.png,Normal,Normal/images/Normal-7221.png,Normal/masks/Normal-7221.png,True,G16520,train,6d526dfcf2e96101ee0315a2b5e72f4e,997b061a5d5a38e9
Normal-7222.png,Normal,Normal/images/Normal-7222.png,Normal/masks/Normal-7222.png,True,G16521,train,88b9f64b3c68a12ba5d9dbced6c7414c,9179861e184f3d79
Normal-7223.png,Normal,Normal/images/Normal-7223.png,Normal/masks/Normal-7223.png,True,G16522,train,95638b9136c2177e569a3c58b470d174,916cc6194d4e794f
Normal-7224.png,Normal,Normal/images/Normal-7224.png,Normal/masks/Normal-7224.png,True,G16523,val,04493d69e4372adcef8e3c5bb5a48fb9,81fbd71a005f5b15
Normal-7225.png,Normal,Normal/images/Normal-7225.png,Normal/masks/Normal-7225.png,True,G16524,val,8fb087cd920ac3e42e185b9cc52b6519,d42d42d70b0c7d6e
Normal-7226.png,Normal,Normal/images/Normal-7226.png,Normal/masks/Normal-7226.png,True,G16525,train,eaf3f1283eefbae20770aae709e66606,997b041e5a4d7978
Normal-7227.png,Normal,Normal/images/Normal-7227.png,Normal/masks/Normal-7227.png,True,G16526,val,54facb0a725d88620298adf629eb6faa,d02e831b0f4e5b4f
Normal-7228.png,Normal,Normal/images/Normal-7228.png,Normal/masks/Normal-7228.png,True,G16527,train,44f18ca1574448e32297f6a32ea7dc79,dd6ed21b0b4c1f09
Normal-7229.png,Normal,Normal/images/Normal-7229.png,Normal/masks/Normal-7229.png,True,G16528,test,a1bae0f356cfab7f3b8dda53966f30bc,8a34b50bfdf4b034
Normal-723.png,Normal,Normal/images/Normal-723.png,Normal/masks/Normal-723.png,True,G16529,test,341b91db96b6635077c3cd682262229d,816fc65ac94d49cd
Normal-7230.png,Normal,Normal/images/Normal-7230.png,Normal/masks/Normal-7230.png,True,G16530,test,47ae49f15403604480cc3c0c9e702097,8968561719574f5b
Normal-7231.png,Normal,Normal/images/Normal-7231.png,Normal/masks/Normal-7231.png,True,G16531,val,2892f2e622323eb57b093d0e2425942e,d97f521b544f1b08
Normal-7232.png,Normal,Normal/images/Normal-7232.png,Normal/masks/Normal-7232.png,True,G16532,train,cea91b166e16d23a52c8a841379e0ec9,c91a461a4e392fb7
Normal-7233.png,Normal,Normal/images/Normal-7233.png,Normal/masks/Normal-7233.png,True,G16533,train,7fd677f95714cc28d2b758969c9bfc93,ca2d8f89a8c2f2d3
Normal-7234.png,Normal,Normal/images/Normal-7234.png,Normal/masks/Normal-7234.png,True,G16534,train,1400408ffabca1f935b60619805a835d,817a5616185d753f
Normal-7235.png,Normal,Normal/images/Normal-7235.png,Normal/masks/Normal-7235.png,True,G16535,train,7bfe26b4bb9e2786214441c11d6310f2,897dc65e104f4d39
Normal-7236.png,Normal,Normal/images/Normal-7236.png,Normal/masks/Normal-7236.png,True,G16536,test,08703bdad8ed2754a751845824be91ee,9939065f1d4f193c
Normal-7237.png,Normal,Normal/images/Normal-7237.png,Normal/masks/Normal-7237.png,True,G16537,train,23a653e583cd888ca86cc904d31d2f0a,d93d421fb96d38a0
Normal-7238.png,Normal,Normal/images/Normal-7238.png,Normal/masks/Normal-7238.png,True,G16538,train,6f2176983c7328d776bb7a4ef26bae47,990d461e471a5b6f
Normal-7239.png,Normal,Normal/images/Normal-7239.png,Normal/masks/Normal-7239.png,True,G16539,test,8e6cc733ea0c4f619fa2c9aac5308403,9168c61a195f79b9
Normal-724.png,Normal,Normal/images/Normal-724.png,Normal/masks/Normal-724.png,True,G16540,train,ba00111e13f43d26c8beb037a86ee876,d03ec75a9d491c9c
Normal-7240.png,Normal,Normal/images/Normal-7240.png,Normal/masks/Normal-7240.png,True,G16541,train,ef9435d84d7e3d7885b3f564fb85db25,d93946971b4c4f19
Normal-7241.png,Normal,Normal/images/Normal-7241.png,Normal/masks/Normal-7241.png,True,G16542,train,2750cc771876b9e05b8740589b018394,dd6d161b494d4e0b
Normal-7242.png,Normal,Normal/images/Normal-7242.png,Normal/masks/Normal-7242.png,True,G16543,train,2fe71aceb5c8932fd6d87bc7439805af,c62c0f2f7d607133
Normal-7243.png,Normal,Normal/images/Normal-7243.png,Normal/masks/Normal-7243.png,True,G16544,train,2c9fd5c4990d49483772f73483563292,cd6d525a1fc10aad
Normal-7244.png,Normal,Normal/images/Normal-7244.png,Normal/masks/Normal-7244.png,True,G16545,train,2cd6d5396f5506d44c5db2bb58fdfa30,d97d469b0bc23a8a
Normal-7245.png,Normal,Normal/images/Normal-7245.png,Normal/masks/Normal-7245.png,True,G16546,train,e28e9a773e8680bf3e2f669dadcd9d95,dd2f565b1a0d6819
Normal-7246.png,Normal,Normal/images/Normal-7246.png,Normal/masks/Normal-7246.png,True,G16547,train,e2e546b58ae0a5b68bcab5b26f4c31da,b91f9e1a59595858
Normal-7247.png,Normal,Normal/images/Normal-7247.png,Normal/masks/Normal-7247.png,True,G16548,val,6097b80d12b985a98e97b16e73b7652d,817d861e590d7e3a
Normal-7248.png,Normal,Normal/images/Normal-7248.png,Normal/masks/Normal-7248.png,True,G16549,val,a49ed886bec1d2c4027c6f7d4bab6395,95281e5b5b397985
Normal-7249.png,Normal,Normal/images/Normal-7249.png,Normal/masks/Normal-7249.png,True,G16550,test,995200bd6be74f0c5a230a407be6589d,9579c6c618194f3b
Normal-725.png,Normal,Normal/images/Normal-725.png,Normal/masks/Normal-725.png,True,G16551,train,14216a2bbde2997ac4ff67bdd6c84267,817cc65ace9f28c5
Normal-7250.png,Normal,Normal/images/Normal-7250.png,Normal/masks/Normal-7250.png,True,G16552,train,b086f486a65b74140ab5b4ac3ab5244c,dd2a530d4c4f4f4a
Normal-7251.png,Normal,Normal/images/Normal-7251.png,Normal/masks/Normal-7251.png,True,G16553,train,2f0d9244cc7968e64c2c9ea6eceb4713,c93fc21b164b1b4d
Normal-7252.png,Normal,Normal/images/Normal-7252.png,Normal/masks/Normal-7252.png,True,G16554,train,cd00ee2cf3cd6489a6184b2ec217a684,d57dc2121b0f0f39
Normal-7253.png,Normal,Normal/images/Normal-7253.png,Normal/masks/Normal-7253.png,True,G16555,train,a1f204d5916756a03845680ca700fc86,ab211e1e39eda9a1
Normal-7254.png,Normal,Normal/images/Normal-7254.png,Normal/masks/Normal-7254.png,True,G16556,train,e2af66db790ca1f92378e2fbbb50e679,992f061a457ab8e7
Normal-7255.png,Normal,Normal/images/Normal-7255.png,Normal/masks/Normal-7255.png,True,G16557,train,66e5d24c8aab659091fbdbc78a0480a6,997b461a1d193d47
Normal-7256.png,Normal,Normal/images/Normal-7256.png,Normal/masks/Normal-7256.png,True,G16558,train,f5d8da7b73125f2fed04de461d9ac102,817f561e180d3f39
Normal-7257.png,Normal,Normal/images/Normal-7257.png,Normal/masks/Normal-7257.png,True,G16559,train,85bf928a294b51b3a0b6e7e372c10b93,c97b05861f185f55
Normal-7258.png,Normal,Normal/images/Normal-7258.png,Normal/masks/Normal-7258.png,True,G16560,train,b5351dff79b917cb161edb691f8f4db3,dc2ec35d0c483dec
Normal-7259.png,Normal,Normal/images/Normal-7259.png,Normal/masks/Normal-7259.png,True,G16561,train,3a877142aa8bb307fbb130998988f1b7,987dc6531b454b39
Normal-726.png,Normal,Normal/images/Normal-726.png,Normal/masks/Normal-726.png,True,G16562,val,e8c745677b25cce295e6091cfe3094bf,917dc61aca1979d4
Normal-7260.png,Normal,Normal/images/Normal-7260.png,Normal/masks/Normal-7260.png,True,G16563,train,c5a7f793d32fb83a7fd4c6d6ebc7b9ab,887f87161b0f1f0e
Normal-7261.png,Normal,Normal/images/Normal-7261.png,Normal/masks/Normal-7261.png,True,G16564,train,adedc0f71e40da03b91eeb3be63e7161,d90e061e595b5c7a
Normal-7262.png,Normal,Normal/images/Normal-7262.png,Normal/masks/Normal-7262.png,True,G16565,train,a9268bd39d63feba7b78ff5316613aba,883c43cb094f6f4f
Normal-7263.png,Normal,Normal/images/Normal-7263.png,Normal/masks/Normal-7263.png,True,G16566,train,0d22ef55a92c34ae864d4fde57965747,9969871a475e18eb
Normal-7264.png,Normal,Normal/images/Normal-7264.png,Normal/masks/Normal-7264.png,True,G16567,val,c17b9adaffb94dd09accd6d8a55beef0,dc2b525b4a4b0b4f
Normal-7265.png,Normal,Normal/images/Normal-7265.png,Normal/masks/Normal-7265.png,True,G16568,val,baf3d514443e47ca4a8bc6468b7ac5ef,997847171a4f593a
Normal-7266.png,Normal,Normal/images/Normal-7266.png,Normal/masks/Normal-7266.png,True,G16569,test,346a8b8fd6198f6d8104212a5f46087e,91785e051959e5d7
Normal-7267.png,Normal,Normal/images/Normal-7267.png,Normal/masks/Normal-7267.png,True,G16570,test,cc52a9b89d220173834d22bd2cba5704,dd6c4f4a4d0e0e4e
Normal-7268.png,Normal,Normal/images/Normal-7268.png,Normal/masks/Normal-7268.png,True,G16571,test,215dafd277ecefe4d0bc97533f753977,99688618175ebbea
Normal-7269.png,Normal,Normal/images/Normal-7269.png,Normal/masks/Normal-7269.png,True,G16572,val,a349d19c3ae6ca057e956d46cfd1a16f,887d871a195f7a29
Normal-727.png,Normal,Normal/images/Normal-727.png,Normal/masks/Normal-727.png,True,G16573,train,cf0065826145c591b39a2936104a9a4f,d074c63e992d3c6c
Normal-7270.png,Normal,Normal/images/Normal-7270.png,Normal/masks/Normal-7270.png,True,G16574,train,75478e54cc93af8e37928a0036fc48bc,d03c4fd91c0fb0b9
Normal-7271.png,Normal,Normal/images/Normal-7271.png,Normal/masks/Normal-7271.png,True,G16575,train,c16522dcd2245fbb7fb3289c254362b7,d949071e0a6939ef
Normal-7272.png,Normal,Normal/images/Normal-7272.png,Normal/masks/Normal-7272.png,True,G16576,train,257f5ce81d1a83f94d025ecb3f9f949d,d93fc61e0d4b194a
Normal-7273.png,Normal,Normal/images/Normal-7273.png,Normal/masks/Normal-7273.png,True,G16577,train,3bc5eeaf12648909d033858dcba187d6,dd2cd05b534a0f4b
Normal-7274.png,Normal,Normal/images/Normal-7274.png,Normal/masks/Normal-7274.png,True,G16578,train,1222915934eb927c346163e4c9bbd1b1,dc2dc2d30b2c8e2f
Normal-7275.png,Normal,Normal/images/Normal-7275.png,Normal/masks/Normal-7275.png,True,G16579,test,20afb5c534dd4bc8feed9bbd429bff93,d82d831d146fbbc2
Normal-7276.png,Normal,Normal/images/Normal-7276.png,Normal/masks/Normal-7276.png,True,G16580,train,e8b387b5774360011715340df7f21e41,dc2f42535b424f1b
Normal-7277.png,Normal,Normal/images/Normal-7277.png,Normal/masks/Normal-7277.png,True,G16581,train,e7ee7b7091de8c3b07a65aedaad66330,f97dc6061f0d0a39
Normal-7278.png,Normal,Normal/images/Normal-7278.png,Normal/masks/Normal-7278.png,True,G16582,train,71ceb865ffa905b240916872d4504061,d928c7961ec7c61a
Normal-7279.png,Normal,Normal/images/Normal-7279.png,Normal/masks/Normal-7279.png,True,G16583,test,dc8994b0bab6010947503842bf77a448,dc3fc308aca9a8a7
Normal-728.png,Normal,Normal/images/Normal-728.png,Normal/masks/Normal-728.png,True,G16584,train,8a5fa23b86854f8e9fd1c9bae249656b,816ae71aedc96cc4
Normal-7280.png,Normal,Normal/images/Normal-7280.png,Normal/masks/Normal-7280.png,True,G16585,train,aca6501b0981d932a6add7e64aea7d58,9968861917793b6e
Normal-7281.png,Normal,Normal/images/Normal-7281.png,Normal/masks/Normal-7281.png,True,G16586,train,45f2c40144b5abcd038dbd2620c3e1e7,d82844d35b554f4f
Normal-7282.png,Normal,Normal/images/Normal-7282.png,Normal/masks/Normal-7282.png,True,G16587,val,549d3d9f76552d70c20c66b8d5811797,bc2ec31b451e4c4f
Normal-7283.png,Normal,Normal/images/Normal-7283.png,Normal/masks/Normal-7283.png,True,G16588,val,ea1b8a43f12fc5ad3180c9270a1263be,987d44561e59e96c
Normal-7284.png,Normal,Normal/images/Normal-7284.png,Normal/masks/Normal-7284.png,True,G16589,test,377ad1cac199bc2039fd16fe98d7c641,9928161a5ad67af9
Normal-7285.png,Normal,Normal/images/Normal-7285.png,Normal/masks/Normal-7285.png,True,G16590,train,178b6dab290eff6fd6d8982afc14ac77,997c4e1bb94cbc14
Normal-7286.png,Normal,Normal/images/Normal-7286.png,Normal/masks/Normal-7286.png,True,G16591,train,b978e5bf6fa5c58cf1433ecc69120152,887d861a1d4f1b5d
Normal-7287.png,Normal,Normal/images/Normal-7287.png,Normal/masks/Normal-7287.png,True,G16592,train,0efdaf3121261159689d3b2a31ce3de1,9978c61a194f38eb
Normal-7288.png,Normal,Normal/images/Normal-7288.png,Normal/masks/Normal-7288.png,True,G16593,train,d1018aa9ea0ba6636c6a147b14878772,816bd61a08c73bf9
Normal-7289.png,Normal,Normal/images/Normal-7289.png,Normal/masks/Normal-7289.png,True,G16594,train,ee676c0abf44369930a5a7cf893a71a2,c03d4f5233f33538
Normal-729.png,Normal,Normal/images/Normal-729.png,Normal/masks/Normal-729.png,True,G16595,train,47302829d16f71cba9ba5f41e6a43929,d464c71eed2d2cc4
Normal-7290.png,Normal,Normal/images/Normal-7290.png,Normal/masks/Normal-7290.png,True,G16596,train,575249dab31be25dce9ee910a9a912f1,d43c43410f4b2f3f
Normal-7291.png,Normal,Normal/images/Normal-7291.png,Normal/masks/Normal-7291.png,True,G16597,train,4c85ed83ce679935d42599cbbf1b15e2,c83d42430f0f6f3d
Normal-7292.png,Normal,Normal/images/Normal-7292.png,Normal/masks/Normal-7292.png,True,G16598,train,b14ce745944ef1677db900b4ce7ed6fc,d03d42071d2ced7d
Normal-7293.png,Normal,Normal/images/Normal-7293.png,Normal/masks/Normal-7293.png,True,G16599,train,8d4e9b94ce5720c305592d9e4231372a,992d865a49557ae5
Normal-7294.png,Normal,Normal/images/Normal-7294.png,Normal/masks/Normal-7294.png,True,G16600,train,52594c0c86461892719cfa41f349d4c5,9d6dc41a5b451e4b
Normal-7295.png,Normal,Normal/images/Normal-7295.png,Normal/masks/Normal-7295.png,True,G16601,train,d743d352378f0035ffe8b2157af0c374,997b46180f3bb86c
Normal-7296.png,Normal,Normal/images/Normal-7296.png,Normal/masks/Normal-7296.png,True,G16602,train,c95f17ec8cd6462490a67fefeaae1aa1,8979861e5b497978
Normal-7297.png,Normal,Normal/images/Normal-7297.png,Normal/masks/Normal-7297.png,True,G16603,train,312107c090273a31027d2bda39e40506,d93d861e1d4e1959
Normal-7298.png,Normal,Normal/images/Normal-7298.png,Normal/masks/Normal-7298.png,True,G16604,train,79d60971a439b3af95d8de382627baf7,997f161a5d3a087a
Normal-7299.png,Normal,Normal/images/Normal-7299.png,Normal/masks/Normal-7299.png,True,G16605,train,4dd34a3db9564d496a8f4c31a62f1f14,d93c421a4b0f1b6f
Normal-73.png,Normal,Normal/images/Normal-73.png,Normal/masks/Normal-73.png,True,G16606,train,c77bcf1c8b43aaaf45886accfeccd24e,d12ace5a4c5d5d45
Normal-730.png,Normal,Normal/images/Normal-730.png,Normal/masks/Normal-730.png,True,G16607,val,cae97bc010f7be81b537382e59a15c24,d46f8f1b8d0d80c7
Normal-7300.png,Normal,Normal/images/Normal-7300.png,Normal/masks/Normal-7300.png,True,G16608,val,696480ce0ade5ac8332404e700d3f865,817b46961859f97a
Normal-7301.png,Normal,Normal/images/Normal-7301.png,Normal/masks/Normal-7301.png,True,G16609,val,95f7d3b7ae13379efb85e6d463c2711c,814d5e5ae9a9e187
Normal-7302.png,Normal,Normal/images/Normal-7302.png,Normal/masks/Normal-7302.png,True,G16610,test,8d584556f8a564cb4f8c7a3dbcf79e23,dc608b1bcf0eb90e
Normal-7303.png,Normal,Normal/images/Normal-7303.png,Normal/masks/Normal-7303.png,True,G16611,test,ba3bb2e0534aee5032857baf0cabcf64,fc38d35b5e52181a
Normal-7304.png,Normal,Normal/images/Normal-7304.png,Normal/masks/Normal-7304.png,True,G16612,test,ba39348ec29b5de62e39993400c5d850,817c06095bede8f9
Normal-7305.png,Normal,Normal/images/Normal-7305.png,Normal/masks/Normal-7305.png,True,G16613,train,70d8a0b6ee3505246083d76c7630d0df,ddacd05bd14b9143
Normal-7306.png,Normal,Normal/images/Normal-7306.png,Normal/masks/Normal-7306.png,True,G16614,train,135ed8b45af69f10b7b0bda6345eb97a,dd2ec0131fcd1a99
Normal-7307.png,Normal,Normal/images/Normal-7307.png,Normal/masks/Normal-7307.png,True,G16615,train,99370891d4c01a278d775e4c98e5678d,d97f061a944a19ed
Normal-7308.png,Normal,Normal/images/Normal-7308.png,Normal/masks/Normal-7308.png,True,G16616,train,50e1e064080c8b916d7dc2c3b5bdf169,993dc61f5c4e3c50
Normal-7309.png,Normal,Normal/images/Normal-7309.png,Normal/masks/Normal-7309.png,True,G16617,train,99803dc7270ecba6f0f3e925c18844ea,803f46df594c4f58
Normal-731.png,Normal,Normal/images/Normal-731.png,Normal/masks/Normal-731.png,True,G16618,train,66665a2c5bc2c22d209105dd4c6b78f3,952f8f1b0d0d0dc7
Normal-7310.png,Normal,Normal/images/Normal-7310.png,Normal/masks/Normal-7310.png,True,G16619,train,749b88c540178d1f612f118b0d63b006,997b975a19047a59
Normal-7311.png,Normal,Normal/images/Normal-7311.png,Normal/masks/Normal-7311.png,True,G16620,val,4ce0a443ed80483cdaa852b1e0852bd8,917d87861b8f8719
Normal-7312.png,Normal,Normal/images/Normal-7312.png,Normal/masks/Normal-7312.png,True,G16621,test,c54946b97ce78d6ca4de872de87a6944,d42e425f0a4e3f6c
Normal-7313.png,Normal,Normal/images/Normal-7313.png,Normal/masks/Normal-7313.png,True,G16622,train,82a21bb875dd9a128a9eabe8f7d6e7e9,dc2ec34f4d460f1c
Normal-7314.png,Normal,Normal/images/Normal-7314.png,Normal/masks/Normal-7314.png,True,G16623,train,4b6fe96b4b1e2d84b9fe4ea3d08cee6b,993d061a1d4e3b7c
Normal-7315.png,Normal,Normal/images/Normal-7315.png,Normal/masks/Normal-7315.png,True,G16624,train,ffab433c34545acf480d4baa8dded337,807f42571f094d7d
Normal-7316.png,Normal,Normal/images/Normal-7316.png,Normal/masks/Normal-7316.png,True,G16625,test,32e3e04f367795edc3460f70b0eeefb0,c13906c71e1b6d9d
Normal-7317.png,Normal,Normal/images/Normal-7317.png,Normal/masks/Normal-7317.png,True,G16626,train,aa7911ab748329483570f5cd7bc02b57,dc3dc30fc2431b4b
Normal-7318.png,Normal,Normal/images/Normal-7318.png,Normal/masks/Normal-7318.png,True,G16627,train,d1231f817d4ed81dc848d90a07d64e77,813196591e1a797f
Normal-7319.png,Normal,Normal/images/Normal-7319.png,Normal/masks/Normal-7319.png,True,G16628,train,e4d78cb35a26171756a356f183022295,d979c45e14471b1d
Normal-732.png,Normal,Normal/images/Normal-732.png,Normal/masks/Normal-732.png,True,G16629,train,e03c2b076c6b0157287ae60c24790295,d475a31fad0c0cc7
Normal-7320.png,Normal,Normal/images/Normal-7320.png,Normal/masks/Normal-7320.png,True,G16630,train,db247b7f5db969722a1948471ef6c66c,993c871b1bc71e19
Normal-7321.png,Normal,Normal/images/Normal-7321.png,Normal/masks/Normal-7321.png,True,G16631,val,209166d4ace21b8ea364e735349f7448,817b8516188ffab9
Normal-7322.png,Normal,Normal/images/Normal-7322.png,Normal/masks/Normal-7322.png,True,G16632,val,27e3d85c7be6d8c8e91c0226a588e530,cd2ec34b5d4a2be0
Normal-7323.png,Normal,Normal/images/Normal-7323.png,Normal/masks/Normal-7323.png,True,G16633,test,38a14fd116dbbdb6a6b5467604e443a0,8178571a5dec39a5
Normal-7324.png,Normal,Normal/images/Normal-7324.png,Normal/masks/Normal-7324.png,True,G16634,train,483fb223ddfcbeb32d0e777a7a9a9dad,c83dc2d65f15c24b
Normal-7325.png,Normal,Normal/images/Normal-7325.png,Normal/masks/Normal-7325.png,True,G16635,train,3f567fd400c894e270e1800c01a433cb,8179c65e1b4f7819
Normal-7326.png,Normal,Normal/images/Normal-7326.png,Normal/masks/Normal-7326.png,True,G16636,train,1f00b88e501977c536bd753da012a688,dd7cc64a1c4e591c
Normal-7327.png,Normal,Normal/images/Normal-7327.png,Normal/masks/Normal-7327.png,True,G16637,train,19901d6ea208c8fcf55807242a11f33c,f96cc41a0d4e1b2f
Normal-7328.png,Normal,Normal/images/Normal-7328.png,Normal/masks/Normal-7328.png,True,G16638,train,0966d3e39a718c8afaf0b12029812778,d13d420b19387b3f
Normal-7329.png,Normal,Normal/images/Normal-7329.png,Normal/masks/Normal-7329.png,True,G16639,train,dc81c7dab0d7e15e6c1bbbc046577653,b91f161819705af7
Normal-733.png,Normal,Normal/images/Normal-733.png,Normal/masks/Normal-733.png,True,G16640,train,d620de2c8056ef914f312ebe0789b219,906dc77ae86969c4
Normal-7330.png,Normal,Normal/images/Normal-7330.png,Normal/masks/Normal-7330.png,True,G16641,train,25318638f76ac631c755c739cdae198b,dd75060a4b7a616e
Normal-7331.png,Normal,Normal/images/Normal-7331.png,Normal/masks/Normal-7331.png,True,G16642,train,83ef6ed59e7a5383c42fd8f0db406fb5,c13c864f194e6d3b
Normal-7332.png,Normal,Normal/images/Normal-7332.png,Normal/masks/Normal-7332.png,True,G16643,train,b32c984ff16e78f2f2d969819567096f,917a065a5b5978f8
Normal-7333.png,Normal,Normal/images/Normal-7333.png,Normal/masks/Normal-7333.png,True,G16644,train,8befa908efc98c45e87cde211e491307,d5691e1a1a1b395b
Normal-7334.png,Normal,Normal/images/Normal-7334.png,Normal/masks/Normal-7334.png,True,G16645,train,8fd8583ae4802562c92c123674fe290f,d97ad2190a393cae
Normal-7335.png,Normal,Normal/images/Normal-7335.png,Normal/masks/Normal-7335.png,True,G16646,val,2ff81d85a5c4855ce9a74b2b04d37181,d52e434f0d4d4e5c
Normal-7336.png,Normal,Normal/images/Normal-7336.png,Normal/masks/Normal-7336.png,True,G16647,train,c99b6f03a323b32a0f11e824fc2b6435,dd2c521bd54b154b
Normal-7337.png,Normal,Normal/images/Normal-7337.png,Normal/masks/Normal-7337.png,True,G16648,val,0e6b03c0b386702f80c92316c032a686,807867865a896fdb
Normal-7338.png,Normal,Normal/images/Normal-7338.png,Normal/masks/Normal-7338.png,True,G16649,test,5c2cfcf5e6b3e11430ee1aa65a1c87d1,d57d434b4b4a4f0a
Normal-7339.png,Normal,Normal/images/Normal-7339.png,Normal/masks/Normal-7339.png,True,G16650,train,f88f88848b570e973ed0d8336d210080,9939461e5a4f3a39
Normal-734.png,Normal,Normal/images/Normal-734.png,Normal/masks/Normal-734.png,True,G16651,val,1a296baf08c54cea73a9afc507b18c1b,dc66871b8f2d0c2e
Normal-7340.png,Normal,Normal/images/Normal-7340.png,Normal/masks/Normal-7340.png,True,G16652,test,0a9831c9d0d772c542b34642ef898413,d0356a0a7f292d3d
Normal-7341.png,Normal,Normal/images/Normal-7341.png,Normal/masks/Normal-7341.png,True,G16653,train,80f0a31b1b4748475fec03f71557b775,bc6cc28bd5aa99a8
Normal-7342.png,Normal,Normal/images/Normal-7342.png,Normal/masks/Normal-7342.png,True,G16654,test,386859b16ae2504d352e3dc77c7f0f9a,dd28071e171e1a5f
Normal-7343.png,Normal,Normal/images/Normal-7343.png,Normal/masks/Normal-7343.png,True,G16655,train,4cb058722c6d7023a52cade30927dd0d,c959161a193b39af
Normal-7344.png,Normal,Normal/images/Normal-7344.png,Normal/masks/Normal-7344.png,True,G16656,train,6aba6e8ec81752674ef00a968221f01e,c120561d7b437b3b
Normal-7345.png,Normal,Normal/images/Normal-7345.png,Normal/masks/Normal-7345.png,True,G16657,train,caad29f2d3695df7fdc62a19526ce45c,d969851a955f1a4d
Normal-7346.png,Normal,Normal/images/Normal-7346.png,Normal/masks/Normal-7346.png,True,G16658,train,279a071d8bb6ee4ed323aee7c70be042,9178ce5a594d5b85
Normal-7347.png,Normal,Normal/images/Normal-7347.png,Normal/masks/Normal-7347.png,True,G16659,test,81c761d2e90559acc87a0feea08e3e9b,9d2ec20b4d4e1d5d
Normal-7348.png,Normal,Normal/images/Normal-7348.png,Normal/masks/Normal-7348.png,True,G16660,test,e32ebb0c6d97b53841f677fb5d1e245f,8968171e1add7a39
Normal-7349.png,Normal,Normal/images/Normal-7349.png,Normal/masks/Normal-7349.png,True,G16661,train,20870ee6b3e8b59b75a6d50e68919725,916806061a5fdf5f
Normal-735.png,Normal,Normal/images/Normal-735.png,Normal/masks/Normal-735.png,True,G16662,train,d60e39fab3c675682ae8388cfd2ae1a0,91abc21ac5b296e7
Normal-7350.png,Normal,Normal/images/Normal-7350.png,Normal/masks/Normal-7350.png,True,G16663,train,89fee0b945de4a7cebdd3a5e9d78a6b5,917bc5061ed54e1e
Normal-7351.png,Normal,Normal/images/Normal-7351.png,Normal/masks/Normal-7351.png,True,G16664,train,60f51646fdc71d2b89be5fdeed3ad8a2,cc2d879e1e8d191d
Normal-7352.png,Normal,Normal/images/Normal-7352.png,Normal/masks/Normal-7352.png,True,G16665,train,f632b3c73294421621346b7a024c052b,f97d065a0cdb18c5
Normal-7353.png,Normal,Normal/images/Normal-7353.png,Normal/masks/Normal-7353.png,True,G16666,train,8a8d7d4e538bb15cf4e581a2fa25e06c,d865075a475a4e4f
Normal-7354.png,Normal,Normal/images/Normal-7354.png,Normal/masks/Normal-7354.png,True,G16667,train,b7f70cf06830f9c3301e8b51d39d1387,996c461a69ed3ae1
Normal-7355.png,Normal,Normal/images/Normal-7355.png,Normal/masks/Normal-7355.png,True,G16668,train,41edd413ce0493b8a324fc2f35279af3,95685e7761a53496
Normal-7356.png,Normal,Normal/images/Normal-7356.png,Normal/masks/Normal-7356.png,True,G16669,train,0c9ab7cdc34d114a82940563d59c6e37,d42d42d75f08ed0b
Normal-7357.png,Normal,Normal/images/Normal-7357.png,Normal/masks/Normal-7357.png,True,G16670,val,a2b34bf67cf78cfc3f986810ba9bdaf1,897a871a1dcf38a9
Normal-7358.png,Normal,Normal/images/Normal-7358.png,Normal/masks/Normal-7358.png,True,G16671,val,cb7954c7e8da46dc15dcc5b6f510dbe2,9d7ec21b492e382d
Normal-7359.png,Normal,Normal/images/Normal-7359.png,Normal/masks/Normal-7359.png,True,G16672,val,4d27b8de4482a6e39c508f6b1da1d5f3,d5790a193d1db8b1
Normal-736.png,Normal,Normal/images/Normal-736.png,Normal/masks/Normal-736.png,True,G16673,test,08f6b45b9ddf3e79ab2e603c18e6a548,997b471e160f3a29
Normal-7360.png,Normal,Normal/images/Normal-7360.png,Normal/masks/Normal-7360.png,True,G16674,train,406772f12058e535ff66e78138958430,d92d461b1d4d1bd4
Normal-7361.png,Normal,Normal/images/Normal-7361.png,Normal/masks/Normal-7361.png,True,G16675,train,906cc00cff7114b193068a913df802d4,dd2cd24b564a1f0b
Normal-7362.png,Normal,Normal/images/Normal-7362.png,Normal/masks/Normal-7362.png,True,G16676,train,889b01c0a02aa3e22d8b3afa86febe30,dd2806931b496ff8
Normal-7363.png,Normal,Normal/images/Normal-7363.png,Normal/masks/Normal-7363.png,True,G16677,train,0659054084744a6077e811c668f48656,81a96ed678936b8a
Normal-7364.png,Normal,Normal/images/Normal-7364.png,Normal/masks/Normal-7364.png,True,G16678,train,a52a42b82373cf6635067924c3ff0d40,987b851e16871fb8
Normal-7365.png,Normal,Normal/images/Normal-7365.png,Normal/masks/Normal-7365.png,True,G16679,train,e1450f3b0c49a6aac3526db1ec656403,c83d854e1f474b1d
Normal-7366.png,Normal,Normal/images/Normal-7366.png,Normal/masks/Normal-7366.png,True,G16680,train,921944f5546bc4c235c530164afa20c4,987d041e1f4e1f5c
Normal-7367.png,Normal,Normal/images/Normal-7367.png,Normal/masks/Normal-7367.png,True,G16681,train,5c76bac7121002dbda6c84d6c1098b3f,cd3ac21f524e1b4e
Normal-7368.png,Normal,Normal/images/Normal-7368.png,Normal/masks/Normal-7368.png,True,G16682,train,158abef23458669a5e1a6471426fd0b2,993d861a98ee7aa8
Normal-7369.png,Normal,Normal/images/Normal-7369.png,Normal/masks/Normal-7369.png,True,G16683,train,e3942575de16dd74e4171fd1d288986a,dd2cd20b0f0c6f1d
Normal-737.png,Normal,Normal/images/Normal-737.png,Normal/masks/Normal-737.png,True,G16684,train,f120bfd3ad188da18304afc682c1347c,806be25a7d793878
Normal-7370.png,Normal,Normal/images/Normal-7370.png,Normal/masks/Normal-7370.png,True,G16685,train,bf41cf372d6c6b07b0bd4ec6d8419e7c,d93846471a8f8f1b
Normal-7371.png,Normal,Normal/images/Normal-7371.png,Normal/masks/Normal-7371.png,True,G16686,train,a325eef7e0734e20c6b113fe87a184ff,d86d841687561bf9
Normal-7372.png,Normal,Normal/images/Normal-7372.png,Normal/masks/Normal-7372.png,True,G16687,val,b7088465f2b5ec45515adfc5d221193c,9d2c061b055f794f
Normal-7373.png,Normal,Normal/images/Normal-7373.png,Normal/masks/Normal-7373.png,True,G16688,val,52fed92bda53843fac8617c0fbf05a60,dd28521353471e5f
Normal-7374.png,Normal,Normal/images/Normal-7374.png,Normal/masks/Normal-7374.png,True,G16689,val,30133f7773236f4423c0b02da4a707d5,d93f861a5e1a585a
Normal-7375.png,Normal,Normal/images/Normal-7375.png,Normal/masks/Normal-7375.png,True,G16690,test,cec6b1bfa467a43880ec0261759ac87d,b96f061e4919535b
Normal-7376.png,Normal,Normal/images/Normal-7376.png,Normal/masks/Normal-7376.png,True,G16691,test,8ac100fbcab04e57ce5cc8ced7ddaa89,dd7ec21b0c6c38e8
Normal-7377.png,Normal,Normal/images/Normal-7377.png,Normal/masks/Normal-7377.png,True,G16692,test,900e35054b48d429a45a1ce65b14e205,994fc71a0d1a68e7
Normal-7378.png,Normal,Normal/images/Normal-7378.png,Normal/masks/Normal-7378.png,True,G16693,train,f161bcb68f4bebfdea0e12460f94d731,9939c61e144f7b58
Normal-7379.png,Normal,Normal/images/Normal-7379.png,Normal/masks/Normal-7379.png,True,G16694,train,a7e7fb35ad2ad3394c1b705060e8deeb,b969c6561f455a1a
Normal-738.png,Normal,Normal/images/Normal-738.png,Normal/masks/Normal-738.png,True,G16695,train,58103b93f00af26712f29980ff7ade11,d1b5c63ae86d3868
Normal-7380.png,Normal,Normal/images/Normal-7380.png,Normal/masks/Normal-7380.png,True,G16696,train,695461e3ad8360ce1dcc7cdcf00a4f42,887c475b5b4d6d18
Normal-7381.png,Normal,Normal/images/Normal-7381.png,Normal/masks/Normal-7381.png,True,G16697,train,b9a3d23870a85b9508a55ef8e06c1e25,996f161a4b595959
Normal-7382.png,Normal,Normal/images/Normal-7382.png,Normal/masks/Normal-7382.png,True,G16698,train,c43fcb0a6ea9fa2c017fa28e8eaf8db4,d93bc6461f1d491a
Normal-7383.png,Normal,Normal/images/Normal-7383.png,Normal/masks/Normal-7383.png,True,G16699,train,80a6f84a8771744fe22e0d29dbbd90b2,d97906161a1ddf1a
Normal-7384.png,Normal,Normal/images/Normal-7384.png,Normal/masks/Normal-7384.png,True,G16700,train,95ea6087b4e7fa51c4af044646f05bb9,807b47961e1a79d9
Normal-7385.png,Normal,Normal/images/Normal-7385.png,Normal/masks/Normal-7385.png,True,G16701,train,bad3f8b8cd531657d48d21c6aae8d985,d938460b1c2bf5b9
Normal-7386.png,Normal,Normal/images/Normal-7386.png,Normal/masks/Normal-7386.png,True,G16702,val,28809121aa833be9ee2fd5defa48cf64,996e061a5d7a70e9
Normal-7387.png,Normal,Normal/images/Normal-7387.png,Normal/masks/Normal-7387.png,True,G16703,train,c02f9c17ababb5dcd5774e41d138a55b,817bc71b104e1d5f
Normal-7388.png,Normal,Normal/images/Normal-7388.png,Normal/masks/Normal-7388.png,True,G16704,train,c803e91480599792c62739f56ed8f47b,9169c68617475e3d
Normal-7389.png,Normal,Normal/images/Normal-7389.png,Normal/masks/Normal-7389.png,True,G16705,val,9a4038e71b3b04ce1c47af30cee92387,dc3cc30b098f2dbc
Normal-739.png,Normal,Normal/images/Normal-739.png,Normal/masks/Normal-739.png,True,G16706,train,500aa7441d12148df98aa285df984d0f,d4cfe319e88cf882
Normal-7390.png,Normal,Normal/images/Normal-7390.png,Normal/masks/Normal-7390.png,True,G16707,train,4c50a56b4471c4333d56227b5b6be859,cd2ed2121f451e5d
Normal-7391.png,Normal,Normal/images/Normal-7391.png,Normal/masks/Normal-7391.png,True,G16708,train,115eaab2d618f4f52aec0462026f3d9d,9969461b5d4f185d
Normal-7392.png,Normal,Normal/images/Normal-7392.png,Normal/masks/Normal-7392.png,True,G16709,train,aa11adf9a75c5458bc6e687a025a7be9,996d46180f4e1beb
Normal-7393.png,Normal,Normal/images/Normal-7393.png,Normal/masks/Normal-7393.png,True,G16710,val,4813937a61ee2abd17e7ac7049d3b369,903d52c7db53c44b
Normal-7394.png,Normal,Normal/images/Normal-7394.png,Normal/masks/Normal-7394.png,True,G16711,test,f026ac258a64e8609f334d3c7d16adde,9168d71e1acd3a39
Normal-7395.png,Normal,Normal/images/Normal-7395.png,Normal/masks/Normal-7395.png,True,G16712,test,3f46887c64e58937d1edde2a6da493cb,893b43471a996d3b
Normal-7396.png,Normal,Normal/images/Normal-7396.png,Normal/masks/Normal-7396.png,True,G16713,test,cc9dbb535d4a664908965d627fc3452f,dc2c53430f5d4e2e
Normal-7397.png,Normal,Normal/images/Normal-7397.png,Normal/masks/Normal-7397.png,True,G16714,train,3081735c690e3bd84f2cebb1914a7a31,d87b851e1a4959f8
Normal-7398.png,Normal,Normal/images/Normal-7398.png,Normal/masks/Normal-7398.png,True,G16715,train,1b72f9718ccb878a79d87948a6849e68,997b471a1c4d3a59
Normal-7399.png,Normal,Normal/images/Normal-7399.png,Normal/masks/Normal-7399.png,True,G16716,train,c565aa0fe86ab4e804c284f5c61df4e2,8c2e425f4d6c6af4
Normal-74.png,Normal,Normal/images/Normal-74.png,Normal/masks/Normal-74.png,True,G16717,train,c9cb76e7691f209427035174833f45eb,90aecb1ecf4c1c47
Normal-740.png,Normal,Normal/images/Normal-740.png,Normal/masks/Normal-740.png,True,G16718,train,f13ea47277853943ffad27be7a3924ca,8185c75ae9c939c7
Normal-7400.png,Normal,Normal/images/Normal-7400.png,Normal/masks/Normal-7400.png,True,G16719,train,7db6623c89ed42a61e92740c2597ba70,dd2c821b454e1f4f
Normal-7401.png,Normal,Normal/images/Normal-7401.png,Normal/masks/Normal-7401.png,True,G16720,train,94bafb853f74ea80a1a6dcaeabb2daac,d92bc21b1d451b3d
Normal-7402.png,Normal,Normal/images/Normal-7402.png,Normal/masks/Normal-7402.png,True,G16721,val,cac3a75df3c0ac8eba7fd67b0e353cd5,8139de1c1aceb9b8
Normal-7403.png,Normal,Normal/images/Normal-7403.png,Normal/masks/Normal-7403.png,True,G16722,train,90a439f938df5595c31103ddb9c5f6ca,d53d42421b056f3f
Normal-7404.png,Normal,Normal/images/Normal-7404.png,Normal/masks/Normal-7404.png,True,G16723,train,9153e6401fee38a8880c607498075296,dc2d82431fd5431f
Normal-7405.png,Normal,Normal/images/Normal-7405.png,Normal/masks/Normal-7405.png,True,G16724,train,c7fc255c8b4410e53bc4e6393c7c7aeb,913a065a5d4b795d
Normal-7406.png,Normal,Normal/images/Normal-7406.png,Normal/masks/Normal-7406.png,True,G16725,train,9ee52a67c49bd134c8ebf96a6fdb2d5a,992d861a5d4e5be4
Normal-7407.png,Normal,Normal/images/Normal-7407.png,Normal/masks/Normal-7407.png,True,G16726,val,209f7ce2527cf8b01037d0ad4a181af9,897c075a1c4d6b3d
Normal-7408.png,Normal,Normal/images/Normal-7408.png,Normal/masks/Normal-7408.png,True,G16727,train,f7346c874188ac7f0e3041dce3989a0d,9978071e1e971a79
Normal-7409.png,Normal,Normal/images/Normal-7409.png,Normal/masks/Normal-7409.png,True,G16728,train,97118e1233180e8a7672127c25927162,d93dc5029747171d
Normal-741.png,Normal,Normal/images/Normal-741.png,Normal/masks/Normal-741.png,True,G16729,train,f8753d1326603d3e76e310ae3fbd38db,913da638b9b129ba
Normal-7410.png,Normal,Normal/images/Normal-7410.png,Normal/masks/Normal-7410.png,True,G16730,val,033aafb076065d7985a4e96dfbd5c621,887d061a134f5e5f
Normal-7411.png,Normal,Normal/images/Normal-7411.png,Normal/masks/Normal-7411.png,True,G16731,test,f701a50c91d58128ece5a4c6e6d132b2,dd4c061a6cecf8c5
Normal-7412.png,Normal,Normal/images/Normal-7412.png,Normal/masks/Normal-7412.png,True,G16732,train,29057342c7cb5d9bf819cf59eee93784,dd2dc21f0d4c1b3c
Normal-7413.png,Normal,Normal/images/Normal-7413.png,Normal/masks/Normal-7413.png,True,G16733,test,abe92f09cfa435082c60b7d145f84c6d,916856195b5f391b
Normal-7414.png,Normal,Normal/images/Normal-7414.png,Normal/masks/Normal-7414.png,True,G16734,train,8d9229759f143e04b2ff0362f5e0d998,91fa87861b17870f
Normal-7415.png,Normal,Normal/images/Normal-7415.png,Normal/masks/Normal-7415.png,True,G16735,test,1bf68104732251eebf43cd5259d9f5af,c93dc21d0c0f3db6
Normal-7416.png,Normal,Normal/images/Normal-7416.png,Normal/masks/Normal-7416.png,True,G16736,train,c144dfc8bbfa06fb2fdefcdc69e0cddc,897805145ff978f1
Normal-7417.png,Normal,Normal/images/Normal-7417.png,Normal/masks/Normal-7417.png,True,G16737,test,602aa1cf2d8e917a6c37b5e996b4563d,990dc716184f4d5f
Normal-7418.png,Normal,Normal/images/Normal-7418.png,Normal/masks/Normal-7418.png,True,G16738,train,26f5a2c206d84464bf37971c7fbbe384,997b071a18cd3bb8
Normal-7419.png,Normal,Normal/images/Normal-7419.png,Normal/masks/Normal-7419.png,True,G16739,train,ead9f547f703538337fd6fcb894182a8,d93b561b184b387a
Normal-742.png,Normal,Normal/images/Normal-742.png,Normal/masks/Normal-742.png,True,G16740,train,6af6f455bd523ecbd4f2dfe62753f864,d4fdc61a8acb38d0
Normal-7420.png,Normal,Normal/images/Normal-7420.png,Normal/masks/Normal-7420.png,True,G16741,train,328cd082937c43eabb9b8d193089a1d7,d93cc653114e0f3b
Normal-7421.png,Normal,Normal/images/Normal-7421.png,Normal/masks/Normal-7421.png,True,G16742,train,6ddc1b059d707733837e67d62fbbdd7a,dc3d420b19a8edb9
Normal-7422.png,Normal,Normal/images/Normal-7422.png,Normal/masks/Normal-7422.png,True,G16743,test,beb102f542c3a9d42b79bb4285a48519,c17956d6585a4d4b
Normal-7423.png,Normal,Normal/images/Normal-7423.png,Normal/masks/Normal-7423.png,True,G16744,test,c36599544022a743d52f8fb13bdd9cea,993b861e544e5a79
Normal-7424.png,Normal,Normal/images/Normal-7424.png,Normal/masks/Normal-7424.png,True,G16745,train,a735a7aa18eb7762c6676579b366b045,9be8851a97c65e94
Normal-7425.png,Normal,Normal/images/Normal-7425.png,Normal/masks/Normal-7425.png,True,G16746,train,dd2bd7e32b2bc1505e25afbfda299a15,8168871e1a5f5f5a
Normal-7426.png,Normal,Normal/images/Normal-7426.png,Normal/masks/Normal-7426.png,True,G16747,train,6567de59f2b042fb0d9b09fbdab8311a,9178c6131bcf5e19
Normal-7427.png,Normal,Normal/images/Normal-7427.png,Normal/masks/Normal-7427.png,True,G16748,train,0dffa73338cb47c48bc075a27524ae82,dc2cd30b5d4c1b5c
Normal-7428.png,Normal,Normal/images/Normal-7428.png,Normal/masks/Normal-7428.png,True,G16749,train,7dd70fff0a43fe74e88a95834579eb94,807d0f5859edb8e1
Normal-7429.png,Normal,Normal/images/Normal-7429.png,Normal/masks/Normal-7429.png,True,G16750,train,c44fb3fa7c11eaded2ada577781f2204,8829475e554f5e78
Normal-743.png,Normal,Normal/images/Normal-743.png,Normal/masks/Normal-743.png,True,G16751,train,0b0414be016a3434f9d2c2f6694a49d2,c12bc65edc692dc1
Normal-7430.png,Normal,Normal/images/Normal-7430.png,Normal/masks/Normal-7430.png,True,G16752,val,27f1ae6096ef76c4b0a041538dbb8470,d93d861e455e1d4c
Normal-7431.png,Normal,Normal/images/Normal-7431.png,Normal/masks/Normal-7431.png,True,G16753,val,42433ce82eb70f678f2f268794ce40de,cc2c424fdc4f9ce8
Normal-7432.png,Normal,Normal/images/Normal-7432.png,Normal/masks/Normal-7432.png,True,G16754,val,1558b83269325ce72a210a74ec667a63,817b841e5a1bdd1b
Normal-7433.png,Normal,Normal/images/Normal-7433.png,Normal/masks/Normal-7433.png,True,G16755,train,257a5892136d888a0c82321f993834a4,d95945161a1f4f1e
Normal-7434.png,Normal,Normal/images/Normal-7434.png,Normal/masks/Normal-7434.png,True,G16756,train,f64b197c99acc9310e8399052b2f3a1a,d93a12931f1e3a1b
Normal-7435.png,Normal,Normal/images/Normal-7435.png,Normal/masks/Normal-7435.png,True,G16757,train,91eed3b2cdb1a0b62f959d4f2c060547,817806095fedeee1
Normal-7436.png,Normal,Normal/images/Normal-7436.png,Normal/masks/Normal-7436.png,True,G16758,train,3d6ecf809c2f0094ee86423b0d842930,993b87169a47ea58
Normal-7437.png,Normal,Normal/images/Normal-7437.png,Normal/masks/Normal-7437.png,True,G16759,train,3cd220001253a34d31fab2e9151fabb9,817e07425b6e6ce9
Normal-7438.png,Normal,Normal/images/Normal-7438.png,Normal/masks/Normal-7438.png,True,G16760,train,e7fd0a1098cbcfa37a9ad1795491fe04,d979c614184e3d1f
Normal-7439.png,Normal,Normal/images/Normal-7439.png,Normal/masks/Normal-7439.png,True,G16761,train,83af6a25ad37f1a4cdbe94b47669204a,913d8656195e4e79
Normal-744.png,Normal,Normal/images/Normal-744.png,Normal/masks/Normal-744.png,True,G16762,train,d47f10c65ecf6a9b490b01df62c6018e,916f873a8f394c4c
Normal-7440.png,Normal,Normal/images/Normal-7440.png,Normal/masks/Normal-7440.png,True,G16763,train,1e3a360ea02462c12dbfd04186f7fe18,8968d75b1ac25f38
Normal-7441.png,Normal,Normal/images/Normal-7441.png,Normal/masks/Normal-7441.png,True,G16764,train,9850e16860ce29ee36447b861b432b9b,893b161e19e9b9a5
Normal-7442.png,Normal,Normal/images/Normal-7442.png,Normal/masks/Normal-7442.png,True,G16765,train,e62b9408a923e2d37b4377d8c83431ba,c17ad3535c4d5958
Normal-7443.png,Normal,Normal/images/Normal-7443.png,Normal/masks/Normal-7443.png,True,G16766,test,e94053d0db243effad04ea9a20fccc02,d12b56161a0cff59
Normal-7444.png,Normal,Normal/images/Normal-7444.png,Normal/masks/Normal-7444.png,True,G16767,train,233e934c6e18bdc024da0a90d0eb169d,d53f024a1e4a5b3e
Normal-7445.png,Normal,Normal/images/Normal-7445.png,Normal/masks/Normal-7445.png,True,G16768,val,0ada65f061e6faa881edbce3ae294090,dd3cc2571bc20e59
Normal-7446.png,Normal,Normal/images/Normal-7446.png,Normal/masks/Normal-7446.png,True,G16769,test,57502d12977e1b7285711b01339e18b9,99ee861f0d180f5d
Normal-7447.png,Normal,Normal/images/Normal-7447.png,Normal/masks/Normal-7447.png,True,G16770,train,31c4d8aef320f8b473137b5769cd33d9,91691e16135e5f78
Normal-7448.png,Normal,Normal/images/Normal-7448.png,Normal/masks/Normal-7448.png,True,G16771,train,f4f0a6e677d30b17439f9e9e3a49967f,9968861a1f1d3d1f
Normal-7449.png,Normal,Normal/images/Normal-7449.png,Normal/masks/Normal-7449.png,True,G16772,val,7690f7ba48d797e4a442e5d5f0682b2e,cc3dc31d4c2f3cc2
Normal-745.png,Normal,Normal/images/Normal-745.png,Normal/masks/Normal-745.png,True,G16773,train,08ec87e82e69783157cbf19f06c1bd7d,c46cc73e8d4d2dc4
Normal-7450.png,Normal,Normal/images/Normal-7450.png,Normal/masks/Normal-7450.png,True,G16774,val,3a44aa8df743036b893e13cd6ad20efb,d93dc61f124e1b92
Normal-7451.png,Normal,Normal/images/Normal-7451.png,Normal/masks/Normal-7451.png,True,G16775,train,f96cff2e46cfb9a819e1e049bcc3aa05,dd2f42d30b0fae0c
Normal-7452.png,Normal,Normal/images/Normal-7452.png,Normal/masks/Normal-7452.png,True,G16776,test,ac2f0bc20260634fde98e590d8775b17,dc24d34bc20f4b4f
Normal-7453.png,Normal,Normal/images/Normal-7453.png,Normal/masks/Normal-7453.png,True,G16777,test,2ab51ac507560f3dd80b0c72f01f6bac,8179875618955e7b
Normal-7454.png,Normal,Normal/images/Normal-7454.png,Normal/masks/Normal-7454.png,True,G16778,train,95b3cfdc2c2f4bd9f09d2d659a7de80e,997c061b0d6f39a5
Normal-7455.png,Normal,Normal/images/Normal-7455.png,Normal/masks/Normal-7455.png,True,G16779,train,551f35d566d2a18f11b6b83c749b07ad,c838d65f94c65ad4
Normal-7456.png,Normal,Normal/images/Normal-7456.png,Normal/masks/Normal-7456.png,True,G16780,train,0227e211cdce9d4a0e24ee0c073fcc21,997d061a1d5b39c5
Normal-7457.png,Normal,Normal/images/Normal-7457.png,Normal/masks/Normal-7457.png,True,G16781,train,0f61e6f26afd43750ac1bc19f3a88cbe,d92ec64a584f5b58
Normal-7458.png,Normal,Normal/images/Normal-7458.png,Normal/masks/Normal-7458.png,True,G16782,train,ed25b8eb607f80a5b1b12f253ca4ef0c,8b78161e78e9e193
Normal-7459.png,Normal,Normal/images/Normal-7459.png,Normal/masks/Normal-7459.png,True,G16783,test,e4dd4a02c512eb7e82115b2f34eaadce,917cc64b1b154e5b
Normal-746.png,Normal,Normal/images/Normal-746.png,Normal/masks/Normal-746.png,True,G16784,train,fef86c79f206b408bfb557607f8d275a,817d871aa96d38e5
Normal-7460.png,Normal,Normal/images/Normal-7460.png,Normal/masks/Normal-7460.png,True,G16785,train,c8b9954460e468e1dd9558f9aa8ee78b,9939e706b247be12
Normal-7461.png,Normal,Normal/images/Normal-7461.png,Normal/masks/Normal-7461.png,True,G16786,train,bc5693cfe7fc3d1e97f6f81fa16b3ad9,807d465f1a4f3959
Normal-7462.png,Normal,Normal/images/Normal-7462.png,Normal/masks/Normal-7462.png,True,G16787,train,7724b0013478433caab311b56df01718,d139821615975f1f
Normal-7463.png,Normal,Normal/images/Normal-7463.png,Normal/masks/Normal-7463.png,True,G16788,train,5c4467ed4b3b776dd8e91594afd7dda3,817b5718501b3b7b
Normal-7464.png,Normal,Normal/images/Normal-7464.png,Normal/masks/Normal-7464.png,True,G16789,train,e2d6238818f7401a808923c7605c8ebc,885d865a59edb9e0
Normal-7465.png,Normal,Normal/images/Normal-7465.png,Normal/masks/Normal-7465.png,True,G16790,val,2987ad65f639d9fa54f967b90914f480,dd2a521f1a0a1d7d
Normal-7466.png,Normal,Normal/images/Normal-7466.png,Normal/masks/Normal-7466.png,True,G16791,val,dcc45d1a1c7faf20879d0b7d397c6773,dc3dc25b1d464f18
Normal-7467.png,Normal,Normal/images/Normal-7467.png,Normal/masks/Normal-7467.png,True,G16792,val,8d91618b113a0261a5b460ab76cda213,982d865a594d78f9
Normal-7468.png,Normal,Normal/images/Normal-7468.png,Normal/masks/Normal-7468.png,True,G16793,test,fe37713f3ae2ee0aeb562864fa39c7ae,913946921bd9eab9
Normal-7469.png,Normal,Normal/images/Normal-7469.png,Normal/masks/Normal-7469.png,True,G16794,train,4ac41c88148ab7aa75450d3087083ea4,cc3dc21f1c4e6b1c
Normal-747.png,Normal,Normal/images/Normal-747.png,Normal/masks/Normal-747.png,True,G16795,train,9e668d7e47cc6f3def60face71d6b5c6,813fc63ab84b39b8
Normal-7470.png,Normal,Normal/images/Normal-7470.png,Normal/masks/Normal-7470.png,True,G16796,train,426bba939fe040b4e5741c122179a44d,dd3cd25f1a4f0c18
Normal-7471.png,Normal,Normal/images/Normal-7471.png,Normal/masks/Normal-7471.png,True,G16797,train,8bcf2a11a06b497646557257bb3ff4c7,997b07161a4dec39
Normal-7472.png,Normal,Normal/images/Normal-7472.png,Normal/masks/Normal-7472.png,True,G16798,train,db7a09b6de3f2d0650ee47eda0b47576,81681e115b7e7979
Normal-7473.png,Normal,Normal/images/Normal-7473.png,Normal/masks/Normal-7473.png,True,G16799,train,09ff4655c561d0e0cf30991d0bd68ad8,8879861e18c5ffb8
Normal-7474.png,Normal,Normal/images/Normal-7474.png,Normal/masks/Normal-7474.png,True,G16800,train,e73abb3469b6ffbf8277122f7712da79,993f065a586e586b
Normal-7475.png,Normal,Normal/images/Normal-7475.png,Normal/masks/Normal-7475.png,True,G16801,train,750a3ff45adb5482d88f17c93e5cff00,dd3f42471b0c4f1a
Normal-7476.png,Normal,Normal/images/Normal-7476.png,Normal/masks/Normal-7476.png,True,G16802,train,d5160e45fe6b5049fb159bfbbcb03eb5,c93dc6431f464b1e
Normal-7477.png,Normal,Normal/images/Normal-7477.png,Normal/masks/Normal-7477.png,True,G16803,train,c94dcc6b04246bca035e6263db393b19,897b061a5bcde839
Normal-7478.png,Normal,Normal/images/Normal-7478.png,Normal/masks/Normal-7478.png,True,G16804,train,ac18f03532d0d8cdaa8ef4efa6cf6892,897946161ec73e99
Normal-7479.png,Normal,Normal/images/Normal-7479.png,Normal/masks/Normal-7479.png,True,G16805,train,7ceb3e17dd3288aa1880aa17189fb78f,d92dc61a5d0c59e9
Normal-748.png,Normal,Normal/images/Normal-748.png,Normal/masks/Normal-748.png,True,G16806,val,3af7bdea126e48d492db760c9f0b2fc7,992fc61acf0d394c
Normal-7480.png,Normal,Normal/images/Normal-7480.png,Normal/masks/Normal-7480.png,True,G16807,train,c3e0cb8dce8342b06944924b8c1cc77c,dc2ed34bc30b4d45
Normal-7481.png,Normal,Normal/images/Normal-7481.png,Normal/masks/Normal-7481.png,True,G16808,val,f27c0ec2c70cdeba0178271cf2583885,d47c06439d1aed2d
Normal-7482.png,Normal,Normal/images/Normal-7482.png,Normal/masks/Normal-7482.png,True,G16809,test,9d85b97634099536847e87291ef49a09,887f071e194f5e19
Normal-7483.png,Normal,Normal/images/Normal-7483.png,Normal/masks/Normal-7483.png,True,G16810,train,af686d919df0d66e605748d0560f2352,c93b061a59e5f079
Normal-7484.png,Normal,Normal/images/Normal-7484.png,Normal/masks/Normal-7484.png,True,G16811,val,f0a8bd92eaeb9c131fab83e4aec1af3f,dc3d421b0b465f5a
Normal-7485.png,Normal,Normal/images/Normal-7485.png,Normal/masks/Normal-7485.png,True,G16812,test,6d7c23b2c1d76a7ec28341dbc82e9b52,9d2cc34b49475a5d
Normal-7486.png,Normal,Normal/images/Normal-7486.png,Normal/masks/Normal-7486.png,True,G16813,test,6c3ae29b7653366a705e8f5e0dd8fd5e,dc2d021f496e78e9
Normal-7487.png,Normal,Normal/images/Normal-7487.png,Normal/masks/Normal-7487.png,True,G16814,train,1ae3deffa824a513868056b575ea8cb7,d939c6d21a5c4e1e
Normal-7488.png,Normal,Normal/images/Normal-7488.png,Normal/masks/Normal-7488.png,True,G16815,train,b9a3e835c85304a6b88fb28e3e2cad75,991907161a5a7f7a
Normal-7489.png,Normal,Normal/images/Normal-7489.png,Normal/masks/Normal-7489.png,True,G16816,train,c1a8bcea9d88af62a274d16ee578d40b,887f57825fd1198c
Normal-749.png,Normal,Normal/images/Normal-749.png,Normal/masks/Normal-749.png,True,G16817,test,303da3b8c20e227e37be5bdc67a1465a,905d861ecd4d8dc7
Normal-7490.png,Normal,Normal/images/Normal-7490.png,Normal/masks/Normal-7490.png,True,G16818,train,f96550d56684624aabfdd53bea453580,9969860a1ad77a79
Normal-7491.png,Normal,Normal/images/Normal-7491.png,Normal/masks/Normal-7491.png,True,G16819,train,13b9045e3f98784aa1bdf1ae166ca291,c9af364072bccd07
Normal-7492.png,Normal,Normal/images/Normal-7492.png,Normal/masks/Normal-7492.png,True,G16820,train,cb9810f7d9cef2e92a30d9bc19b76c51,9a658498d9b1cd6d
Normal-7493.png,Normal,Normal/images/Normal-7493.png,Normal/masks/Normal-7493.png,True,G16821,train,b3e5a3d16e431ac6eaa42df5a39efceb,d97bc64e1a454b38
Normal-7494.png,Normal,Normal/images/Normal-7494.png,Normal/masks/Normal-7494.png,True,G16822,test,febcb572825bf8fb8ef35d365f503b0f,897a971a191f78a5
Normal-7495.png,Normal,Normal/images/Normal-7495.png,Normal/masks/Normal-7495.png,True,G16823,train,78ca6b80fa60ea5359f9eb7cedf5abc1,dd2a534b4c4b1d4b
Normal-7496.png,Normal,Normal/images/Normal-7496.png,Normal/masks/Normal-7496.png,True,G16824,train,88de7225178fed815593fff19515f85f,dd2ad64b1f0c6b18
Normal-7497.png,Normal,Normal/images/Normal-7497.png,Normal/masks/Normal-7497.png,True,G16825,train,6b168b99c8779e419d881991e2dde34a,997d445b194f1f18
Normal-7498.png,Normal,Normal/images/Normal-7498.png,Normal/masks/Normal-7498.png,True,G16826,train,b7662ad69ff2527da427fd2027f62cd5,997b570a1a1de938
Normal-7499.png,Normal,Normal/images/Normal-7499.png,Normal/masks/Normal-7499.png,True,G16827,val,0001babf4b418bc80722c4ca94b3e03d,de2d51534f474a0e
Normal-75.png,Normal,Normal/images/Normal-75.png,Normal/masks/Normal-75.png,True,G16828,train,ee7752cce599a61379a9b7d38881db03,c82bd65edf4815c2
Normal-750.png,Normal,Normal/images/Normal-750.png,Normal/masks/Normal-750.png,True,G16829,train,88f1b5dc088538631f47529b4ca7a690,95b8c71b9dce3890
Normal-7500.png,Normal,Normal/images/Normal-7500.png,Normal/masks/Normal-7500.png,True,G16830,train,636221b7bfc539cb7c3992c153f6a7b5,d96946d65bc24c5a
Normal-7501.png,Normal,Normal/images/Normal-7501.png,Normal/masks/Normal-7501.png,True,G16831,val,190650abcb3e5d140014ce5c0cd42b7f,fc0cc30b2ce4b9d5
Normal-7502.png,Normal,Normal/images/Normal-7502.png,Normal/masks/Normal-7502.png,True,G16832,val,6e9fac46d46b8bb90bfc4ef81fdd079d,9979c71e124f1a39
Normal-7503.png,Normal,Normal/images/Normal-7503.png,Normal/masks/Normal-7503.png,True,G16833,val,c20d7bf3da280560636507dbd70a8059,912ac61a5b4e7a7a
Normal-7504.png,Normal,Normal/images/Normal-7504.png,Normal/masks/Normal-7504.png,True,G16834,train,c5ae58d63b8133a910fb39593502f7ec,dc3cc21b1b8a2bbc
Normal-7505.png,Normal,Normal/images/Normal-7505.png,Normal/masks/Normal-7505.png,True,G16835,test,6a1d654e7f3ae54590a13ae41eaa974a,997dc61a190f3ca9
Normal-7506.png,Normal,Normal/images/Normal-7506.png,Normal/masks/Normal-7506.png,True,G16836,train,4984d61c135b2b8183b8d507f9edc7ec,d82c461a596f78f8
Normal-7507.png,Normal,Normal/images/Normal-7507.png,Normal/masks/Normal-7507.png,True,G16837,train,81c9481c4a035204f500aca5038f799f,997b851e190d5f19
Normal-7508.png,Normal,Normal/images/Normal-7508.png,Normal/masks/Normal-7508.png,True,G16838,train,cc788f3c45988c63383a9772cd2d1de4,dd2b521f0c493dac
Normal-7509.png,Normal,Normal/images/Normal-7509.png,Normal/masks/Normal-7509.png,True,G16839,train,fbc2274ab2c73cef9d135a366766daba,9979c5161ac55f19
Normal-751.png,Normal,Normal/images/Normal-751.png,Normal/masks/Normal-751.png,True,G16840,train,c492957adf5ba4528799736d66885307,d46dc71e892d7894
Normal-7510.png,Normal,Normal/images/Normal-7510.png,Normal/masks/Normal-7510.png,True,G16841,train,055f75c5f224ce2d0faebca032a37fb2,9c2dc2d31f4d451e
Normal-7511.png,Normal,Normal/images/Normal-7511.png,Normal/masks/Normal-7511.png,True,G16842,train,2934a94704fdc282fe4c16a79a645d77,8169961a195d7bb9
Normal-7512.png,Normal,Normal/images/Normal-7512.png,Normal/masks/Normal-7512.png,True,G16843,train,826133c5a7479aa81c33847c06fbd0d8,9c2cc31b1f451d5d
Normal-7513.png,Normal,Normal/images/Normal-7513.png,Normal/masks/Normal-7513.png,True,G16844,train,63c4848f2688681ce471501be1777186,8c3e83d70b151f0f
Normal-7514.png,Normal,Normal/images/Normal-7514.png,Normal/masks/Normal-7514.png,True,G16845,train,2057d58a291aa7b7ab5a53d5397d49e8,d8bdc25f90cb1b90
Normal-7515.png,Normal,Normal/images/Normal-7515.png,Normal/masks/Normal-7515.png,True,G16846,train,f7aeb8a6d62bf6415ed9437c4b4e6adb,982a575b594d38f8
Normal-7516.png,Normal,Normal/images/Normal-7516.png,Normal/masks/Normal-7516.png,True,G16847,train,1e602c93a025958d34a2f7292caea992,d92947021f5f4f19
Normal-7517.png,Normal,Normal/images/Normal-7517.png,Normal/masks/Normal-7517.png,True,G16848,val,5106859021fe6a17d18f6db85de8bab9,895e165a5d7849f8
Normal-7518.png,Normal,Normal/images/Normal-7518.png,Normal/masks/Normal-7518.png,True,G16849,train,60349fbca4e22d224a4d2d6e6e6a7fe2,f969e7c61a180c4f
Normal-7519.png,Normal,Normal/images/Normal-7519.png,Normal/masks/Normal-7519.png,True,G16850,val,cbd20f4fc996ab20da060ed6398442f5,996916185f1d1a5f
Normal-752.png,Normal,Normal/images/Normal-752.png,Normal/masks/Normal-752.png,True,G16851,val,b52693531c88598527a7890f790c26aa,816f471a4f4f58c5
Normal-7520.png,Normal,Normal/images/Normal-7520.png,Normal/masks/Normal-7520.png,True,G16852,test,87c7d637554ba14ad8d82b2262a894bf,887d47094d4e7d47
Normal-7521.png,Normal,Normal/images/Normal-7521.png,Normal/masks/Normal-7521.png,True,G16853,test,0b043e8ee3c0e03f60a889caf8502066,817b57061a175f1e
Normal-7522.png,Normal,Normal/images/Normal-7522.png,Normal/masks/Normal-7522.png,True,G16854,test,e580a0a7751d72c25de1e7c25e490eb5,d8ad061b071e5b4f
Normal-7523.png,Normal,Normal/images/Normal-7523.png,Normal/masks/Normal-7523.png,True,G16855,val,a9e528115a47fe36250daa9ec1f16271,d13a161a1a7b78f8
Normal-7524.png,Normal,Normal/images/Normal-7524.png,Normal/masks/Normal-7524.png,True,G16856,train,8f2fc809c17929a8f5ed51f1358a394d,dc2dc31d0a2d39e6
Normal-7525.png,Normal,Normal/images/Normal-7525.png,Normal/masks/Normal-7525.png,True,G16857,train,6ae4ab0952602ad02d716765e2364e52,996956189a4f3bb1
Normal-7526.png,Normal,Normal/images/Normal-7526.png,Normal/masks/Normal-7526.png,True,G16858,train,ab7be9eb6b1e433bcbd87b2ea8e69e7b,887905161e5f7ab9
Normal-7527.png,Normal,Normal/images/Normal-7527.png,Normal/masks/Normal-7527.png,True,G16859,train,d7ba31360e0f0645d87fc7d6f62ee141,85780ecf9a98e535
Normal-7528.png,Normal,Normal/images/Normal-7528.png,Normal/masks/Normal-7528.png,True,G16860,train,3f4af8aad5dd253b0e197849a2b1b2e1,987b071a157a7ae8
Normal-7529.png,Normal,Normal/images/Normal-7529.png,Normal/masks/Normal-7529.png,True,G16861,train,efa552f188e500228cdf14b1675c90b0,dd2a56d25a1f4a49
Normal-753.png,Normal,Normal/images/Normal-753.png,Normal/masks/Normal-753.png,True,G16862,train,a8c98a96df309d23e5f70a06492eabba,983dc71a9f491ccc
Normal-7530.png,Normal,Normal/images/Normal-7530.png,Normal/masks/Normal-7530.png,True,G16863,train,653933ad361fcca5f2461cf34b006614,d039430f1fbcec19
Normal-7531.png,Normal,Normal/images/Normal-7531.png,Normal/masks/Normal-7531.png,True,G16864,train,5b956ae0a73cf1c32605de02866e129c,ddb4d249d3424b53
Normal-7532.png,Normal,Normal/images/Normal-7532.png,Normal/masks/Normal-7532.png,True,G16865,train,b736ddc792b76d407ddbc1bed1fe71a0,b91d461a417969eb
Normal-7533.png,Normal,Normal/images/Normal-7533.png,Normal/masks/Normal-7533.png,True,G16866,train,38f22d60ff63b2cfb5ecbe7ecb05c0d8,d43dc30b0d8e2cbd
Normal-7534.png,Normal,Normal/images/Normal-7534.png,Normal/masks/Normal-7534.png,True,G16867,train,53eda51c0c7cdb2f758cf9fc1d930201,997dc65e184d6a19
Normal-7535.png,Normal,Normal/images/Normal-7535.png,Normal/masks/Normal-7535.png,True,G16868,train,8d48a59a5278547d53f040a6a9e8c5a9,d979441e1d0d5b1d
Normal-7536.png,Normal,Normal/images/Normal-7536.png,Normal/masks/Normal-7536.png,True,G16869,val,53b455d2e10c41ebe545962ea7ef87a6,817a171b391b39bc
Normal-7537.png,Normal,Normal/images/Normal-7537.png,Normal/masks/Normal-7537.png,True,G16870,train,3ab9c2b5a672421734d534ebb43e2b67,9979071a996978e9
Normal-7538.png,Normal,Normal/images/Normal-7538.png,Normal/masks/Normal-7538.png,True,G16871,train,1bb768734955d7bcfbdd47cf74842b4c,c1395e1a7b1a5a78
Normal-7539.png,Normal,Normal/images/Normal-7539.png,Normal/masks/Normal-7539.png,True,G16872,val,d7dfbf0e94f3c669845893f592c3e12c,d13dc61a19477f18
Normal-754.png,Normal,Normal/images/Normal-754.png,Normal/masks/Normal-754.png,True,G16873,test,e4a9374439ab0238d8482dc198dd1d14,913bde5e5c591941
Normal-7540.png,Normal,Normal/images/Normal-7540.png,Normal/masks/Normal-7540.png,True,G16874,test,c0a54a6de82f68faa723d99d37e04086,d12f0257174e1b5d
Normal-7541.png,Normal,Normal/images/Normal-7541.png,Normal/masks/Normal-7541.png,True,G16875,test,193df663333034015a0afbccae65ddf4,dd28c61319ef0e39
Normal-7542.png,Normal,Normal/images/Normal-7542.png,Normal/masks/Normal-7542.png,True,G16876,train,91b003273a47e50b1206545e24aed537,f96d861e18451b5d
Normal-7543.png,Normal,Normal/images/Normal-7543.png,Normal/masks/Normal-7543.png,True,G16877,train,0dc4f5dea911a35a7995108b84445c47,dd2cc21bd5471a6c
Normal-7544.png,Normal,Normal/images/Normal-7544.png,Normal/masks/Normal-7544.png,True,G16878,train,629bd98526132201a695c6ad74aa48e2,dc3dc30b1f0a2f2a
Normal-7545.png,Normal,Normal/images/Normal-7545.png,Normal/masks/Normal-7545.png,True,G16879,train,bd13de3b5878066f2c86bf82b65d7eee,dd0a530f6c3a2d2e
Normal-7546.png,Normal,Normal/images/Normal-7546.png,Normal/masks/Normal-7546.png,True,G16880,train,8e7212bffd777e26a571fe4d1479b4cc,997846194a6fb969
Normal-7547.png,Normal,Normal/images/Normal-7547.png,Normal/masks/Normal-7547.png,True,G16881,train,9d90faee3144f0be51deb56dc3d9a8c6,807f561a5d2f3147
Normal-7548.png,Normal,Normal/images/Normal-7548.png,Normal/masks/Normal-7548.png,True,G16882,train,05b13b85e673fdb37ffc324c5d1c0e73,d1795e1e190a39b9
Normal-7549.png,Normal,Normal/images/Normal-7549.png,Normal/masks/Normal-7549.png,True,G16883,train,4dd798815620edbf5343fb3dfc09660e,d129de0a590ff929
Normal-755.png,Normal,Normal/images/Normal-755.png,Normal/masks/Normal-755.png,True,G16884,train,89d9f7db1271990ea0547ceae4b1be75,9164871acd6d39c7
Normal-7550.png,Normal,Normal/images/Normal-7550.png,Normal/masks/Normal-7550.png,True,G16885,train,294161fe2b750047773695a41c62b0bb,d93d46031bc64f3a
Normal-7551.png,Normal,Normal/images/Normal-7551.png,Normal/masks/Normal-7551.png,True,G16886,val,76060c12f1ca8c73f9326b47cb12e7fc,dd29d31b494d1959
Normal-7552.png,Normal,Normal/images/Normal-7552.png,Normal/masks/Normal-7552.png,True,G16887,train,e9ff388662f02a16d19b0f8f06881731,d13c43471b0e6d1f
Normal-7553.png,Normal,Normal/images/Normal-7553.png,Normal/masks/Normal-7553.png,True,G16888,train,9ef61abd2da3a09a371877fcaa615b06,813986c61b1fc74f
Normal-7554.png,Normal,Normal/images/Normal-7554.png,Normal/masks/Normal-7554.png,True,G16889,train,858a99abec79d352f72d8ef3ee746d95,dd3d42191a4d39e5
Normal-7555.png,Normal,Normal/images/Normal-7555.png,Normal/masks/Normal-7555.png,True,G16890,train,03909cb7490641caa1b937ecfaae41f6,c97c460a1bdf6c38
Normal-7556.png,Normal,Normal/images/Normal-7556.png,Normal/masks/Normal-7556.png,True,G16891,train,9201cea3bdc5e6b41d9b4acea02fc506,c53e520f5a4e0b5b
Normal-7557.png,Normal,Normal/images/Normal-7557.png,Normal/masks/Normal-7557.png,True,G16892,test,1cef0ad7edf927e5050715ff22e56169,dc3cc25b57461b0e
Normal-7558.png,Normal,Normal/images/Normal-7558.png,Normal/masks/Normal-7558.png,True,G16893,train,e478e8fc4821bdeb26951f40940ed402,dd3d821fd44b194a
Normal-7559.png,Normal,Normal/images/Normal-7559.png,Normal/masks/Normal-7559.png,True,G16894,train,4e6fe0a2b1bb539639357924813e12b4,912a861a5f5f194f
Normal-756.png,Normal,Normal/images/Normal-756.png,Normal/masks/Normal-756.png,True,G16895,val,443571f56760ae392b2a8a638dca6ce9,d0ecc739cc6c38c6
Normal-7560.png,Normal,Normal/images/Normal-7560.png,Normal/masks/Normal-7560.png,True,G16896,val,fcb222b4454aaad570c30d1373f6c6be,d93cc6d316c21b5a
Normal-7561.png,Normal,Normal/images/Normal-7561.png,Normal/masks/Normal-7561.png,True,G16897,test,8c14411aac98981d03fc7db3e4f0c612,9969861a4f7e7849
Normal-7562.png,Normal,Normal/images/Normal-7562.png,Normal/masks/Normal-7562.png,True,G16898,train,12bba3103be8500928080088aa19ebd7,d939c41f96441f95
Normal-7563.png,Normal,Normal/images/Normal-7563.png,Normal/masks/Normal-7563.png,True,G16899,train,f39dfc2e5616c5848c59578b0f9b83c5,993d86185959796d
Normal-7564.png,Normal,Normal/images/Normal-7564.png,Normal/masks/Normal-7564.png,True,G16900,test,c859725810b9c90038d8016b59f11819,887d47961ec94f98
Normal-7565.png,Normal,Normal/images/Normal-7565.png,Normal/masks/Normal-7565.png,True,G16901,test,d63c094bd2ea53cd701a28a5ad14eaf6,917b061a19ede9b1
Normal-7566.png,Normal,Normal/images/Normal-7566.png,Normal/masks/Normal-7566.png,True,G16902,train,179e8d13c1a0102131f36f716e958905,d43c434d0b4a6d7d
Normal-7567.png,Normal,Normal/images/Normal-7567.png,Normal/masks/Normal-7567.png,True,G16903,train,0b5d7199317524fec6164b17f2142317,cc300b4f199de73c
Normal-7568.png,Normal,Normal/images/Normal-7568.png,Normal/masks/Normal-7568.png,True,G16904,train,a41d65bb67c97256797ad58ee4b41a2a,dc35431b1f822db9
Normal-7569.png,Normal,Normal/images/Normal-7569.png,Normal/masks/Normal-7569.png,True,G16905,test,03e607b5a15b0889108c0b7c430f7dad,a87dd71b1d4d1988
Normal-757.png,Normal,Normal/images/Normal-757.png,Normal/masks/Normal-757.png,True,G16906,train,8dfae1323446f8901573a774dcbe9fea,903ec638ed6d8cc6
Normal-7570.png,Normal,Normal/images/Normal-7570.png,Normal/masks/Normal-7570.png,True,G16907,train,6619fdeb6707525314d377e09138ae70,c625434b0e6e2eee
Normal-7571.png,Normal,Normal/images/Normal-7571.png,Normal/masks/Normal-7571.png,True,G16908,train,9223c2f9a7162fb02d2c65c4aa892fb7,d9691617191a6f39
Normal-7572.png,Normal,Normal/images/Normal-7572.png,Normal/masks/Normal-7572.png,True,G16909,train,3413f6fbcae9140db238634d5b45081c,897b871a155e5a59
Normal-7573.png,Normal,Normal/images/Normal-7573.png,Normal/masks/Normal-7573.png,True,G16910,train,fa6364e7991da4695f926b5b66d49950,d03c0fc30b0baecf
Normal-7574.png,Normal,Normal/images/Normal-7574.png,Normal/masks/Normal-7574.png,True,G16911,train,9a9e2d662869615bd5cb70048354f769,dd3bc61a186c2ab9
Normal-7575.png,Normal,Normal/images/Normal-7575.png,Normal/masks/Normal-7575.png,True,G16912,test,ce3c12a3e0616c5fd615895196a26a38,897c8712175f5aa5
Normal-7576.png,Normal,Normal/images/Normal-7576.png,Normal/masks/Normal-7576.png,True,G16913,train,8d82bfd3fe2c6f6134bf5732aea44b18,dd2cc20f544b1f4d
Normal-7577.png,Normal,Normal/images/Normal-7577.png,Normal/masks/Normal-7577.png,True,G16914,train,45bb94cab6feb76c408ea79ce54ae722,cd3c821f1e4b1a5d
Normal-7578.png,Normal,Normal/images/Normal-7578.png,Normal/masks/Normal-7578.png,True,G16915,train,4ad5e3c182b62895f9145b6db20e3b65,917b061e151f1d5d
Normal-7579.png,Normal,Normal/images/Normal-7579.png,Normal/masks/Normal-7579.png,True,G16916,val,05ce7c8a3d92bb1a607ccf2874897866,9978051e1f077ad9
Normal-758.png,Normal,Normal/images/Normal-758.png,Normal/masks/Normal-758.png,True,G16917,val,28ca0cdd7ebf8b28f05e14b20d10678f,916bc63ac97918d6
Normal-7580.png,Normal,Normal/images/Normal-7580.png,Normal/masks/Normal-7580.png,True,G16918,val,b52b5b2e967f4f5f08441957c01a07c9,9c6c170b296c6c7e
Normal-7581.png,Normal,Normal/images/Normal-7581.png,Normal/masks/Normal-7581.png,True,G16919,train,64991031f82907400035d1f617081d1e,8168975a1b8f3b39
Normal-7582.png,Normal,Normal/images/Normal-7582.png,Normal/masks/Normal-7582.png,True,G16920,train,754f0c7df2a37a72bc2fb7a443b5429f,dc3dc2530b4b0f4b
Normal-7583.png,Normal,Normal/images/Normal-7583.png,Normal/masks/Normal-7583.png,True,G16921,train,b2dcc3af2c2fe350098d6b3102ff935b,8c2ed713094f0f1f
Normal-7584.png,Normal,Normal/images/Normal-7584.png,Normal/masks/Normal-7584.png,True,G16922,train,636ae29855187f3c4e06c2df0fc885ee,986f871a0d1f491f
Normal-7585.png,Normal,Normal/images/Normal-7585.png,Normal/masks/Normal-7585.png,True,G16923,train,6242289c8a92a70084a9103c7bc52781,9178965fb85da911
Normal-7586.png,Normal,Normal/images/Normal-7586.png,Normal/masks/Normal-7586.png,True,G16924,train,2153e111c6726d7e4be4a36486bbadfb,885f041c5b7a7ae9
Normal-7587.png,Normal,Normal/images/Normal-7587.png,Normal/masks/Normal-7587.png,True,G16925,train,1d46b7c5d88e2708f198843e9d300bdb,cc3cb543bb0ac30f
Normal-7588.png,Normal,Normal/images/Normal-7588.png,Normal/masks/Normal-7588.png,True,G16926,train,41a4caa5485bedc9824dac926b8c39c7,d17d06080e6f7a4f
Normal-7589.png,Normal,Normal/images/Normal-7589.png,Normal/masks/Normal-7589.png,True,G16927,train,5022cbe8712aaf687830f75831a2e685,d52a42df185d0f1d
Normal-759.png,Normal,Normal/images/Normal-759.png,Normal/masks/Normal-759.png,True,G16928,val,0ac67c647abcc22470323c626b86dd17,916dc71e9a4939d1
Normal-7590.png,Normal,Normal/images/Normal-7590.png,Normal/masks/Normal-7590.png,True,G16929,test,ed92ff22b065dfbba346c65bd0cf18ae,986fc75b4d1a090f
Normal-7591.png,Normal,Normal/images/Normal-7591.png,Normal/masks/Normal-7591.png,True,G16930,train,ed8766c0905b8b29daf7fc286c9f1c5f,c93dd20f544b1f49
Normal-7592.png,Normal,Normal/images/Normal-7592.png,Normal/masks/Normal-7592.png,True,G16931,train,67bb87c4707912e556eb8578c44550dd,d03c4f43190f2f6d
Normal-7593.png,Normal,Normal/images/Normal-7593.png,Normal/masks/Normal-7593.png,True,G16932,train,89623d5d6139aaabe3e6399a1b64c22f,c93d12830b9de8ee
Normal-7594.png,Normal,Normal/images/Normal-7594.png,Normal/masks/Normal-7594.png,True,G16933,test,ce1ea6c5d3dd48402be07569010d321f,892f468e19edaaf0
Normal-7595.png,Normal,Normal/images/Normal-7595.png,Normal/masks/Normal-7595.png,True,G16934,test,01ca5665359f6470bcbc7fa0200a03fa,d83dc6531fc2191e
Normal-7596.png,Normal,Normal/images/Normal-7596.png,Normal/masks/Normal-7596.png,True,G16935,train,f136c8bf4f5082635335ebcc0fc86fd4,817d86161a1f7f38
Normal-7597.png,Normal,Normal/images/Normal-7597.png,Normal/masks/Normal-7597.png,True,G16936,train,b62bad09ef8cc0b6bbe79c17b4fd335b,dd7c87821b0fc64a
Normal-7598.png,Normal,Normal/images/Normal-7598.png,Normal/masks/Normal-7598.png,True,G16937,val,eff483f5d164d230a09ad5a54d844aab,d03d6ac2d71a554b
Normal-7599.png,Normal,Normal/images/Normal-7599.png,Normal/masks/Normal-7599.png,True,G16938,val,be2a2b86289286e45b6ac626ff48e17c,d4bc33c32bca2bc2
Normal-76.png,Normal,Normal/images/Normal-76.png,Normal/masks/Normal-76.png,True,G16939,train,db4c1293bad11ed9750a180ff62c0187,9179ee3a694939c8
Normal-760.png,Normal,Normal/images/Normal-760.png,Normal/masks/Normal-760.png,True,G16940,train,21aceed54c12e80457d4c97e21e8e4e9,d13dc21bbc693c70
Normal-7600.png,Normal,Normal/images/Normal-7600.png,Normal/masks/Normal-7600.png,True,G16941,train,dbafcdb344c195269526dc0dd8506e65,987984161e4d5f3b
Normal-7601.png,Normal,Normal/images/Normal-7601.png,Normal/masks/Normal-7601.png,True,G16942,train,da324bbb4cc699c107f580d4fcb01507,9f68865ab3436333
Normal-7602.png,Normal,Normal/images/Normal-7602.png,Normal/masks/Normal-7602.png,True,G16943,train,dc1db0d558c94eaa131f7b37be23d42c,993b071e1e4ceab8
Normal-7603.png,Normal,Normal/images/Normal-7603.png,Normal/masks/Normal-7603.png,True,G16944,train,9a5320fe1f8854223f4e278a64e600eb,983d421f4e4b1d4f
Normal-7604.png,Normal,Normal/images/Normal-7604.png,Normal/masks/Normal-7604.png,True,G16945,train,5b7190e0a42249947951628b9e8298d7,9968065b194f6f3c
Normal-7605.png,Normal,Normal/images/Normal-7605.png,Normal/masks/Normal-7605.png,True,G16946,train,fe6a03d8d32233f31d874b65ff4b4e22,dcaec259d54f3c40
Normal-7606.png,Normal,Normal/images/Normal-7606.png,Normal/masks/Normal-7606.png,True,G16947,train,ced8bbf6031435c4a8771cb5b2ded583,916906465e9ecf4b
Normal-7607.png,Normal,Normal/images/Normal-7607.png,Normal/masks/Normal-7607.png,True,G16948,val,a5561de29c36f6f6c3509254ac7eb36f,d4bd2bc22d482d6b
Normal-7608.png,Normal,Normal/images/Normal-7608.png,Normal/masks/Normal-7608.png,True,G16949,train,fd638efceeab60ac426606c8fc65379f,cc2c424b4f4f0f4f
Normal-7609.png,Normal,Normal/images/Normal-7609.png,Normal/masks/Normal-7609.png,True,G16950,val,74365909b0783edab98965868752b9be,80fdc71f15060fb5
Normal-761.png,Normal,Normal/images/Normal-761.png,Normal/masks/Normal-761.png,True,G16951,train,6d7c95130d7256695b003642f267ef7e,942b831add1e38d7
Normal-7610.png,Normal,Normal/images/Normal-7610.png,Normal/masks/Normal-7610.png,True,G16952,train,a4d8eed71dba56ed328c98cd4952d720,891d061e1d5f1a5f
Normal-7611.png,Normal,Normal/images/Normal-7611.png,Normal/masks/Normal-7611.png,True,G16953,train,8a0af3ce6a188b4c266d01294d8b59a8,cc3d430a0e6d3df8
Normal-7612.png,Normal,Normal/images/Normal-7612.png,Normal/masks/Normal-7612.png,True,G16954,test,654fc642f1ce1456fbadee07c6e8b8f1,dd2fc21349471a4f
Normal-7613.png,Normal,Normal/images/Normal-7613.png,Normal/masks/Normal-7613.png,True,G16955,val,df4c7e752ded7056b5f457a361894d6c,dcacc61bd50a3c4d
Normal-7614.png,Normal,Normal/images/Normal-7614.png,Normal/masks/Normal-7614.png,True,G16956,test,879be35c380389852d370a306f30c98c,912986961f1fba1b
Normal-7615.png,Normal,Normal/images/Normal-7615.png,Normal/masks/Normal-7615.png,True,G16957,test,5d208e2bf4d3851834a03a18aa122f6e,d42cc3f34b464f58
Normal-7616.png,Normal,Normal/images/Normal-7616.png,Normal/masks/Normal-7616.png,True,G16958,train,f27e2d42dad0b23b82a35620f7bbf0c1,d97906161e5e1b59
Normal-7617.png,Normal,Normal/images/Normal-7617.png,Normal/masks/Normal-7617.png,True,G16959,train,501cd53d0fc127e8d2def1f7d7fcae69,dc2c42431c4f5f5d
Normal-7618.png,Normal,Normal/images/Normal-7618.png,Normal/masks/Normal-7618.png,True,G16960,train,b05c9d6288b7a04b872733bde95fa6b2,dd2c420b5f4a5a7a
Normal-7619.png,Normal,Normal/images/Normal-7619.png,Normal/masks/Normal-7619.png,True,G16961,train,ed86f4875de61cc7e3868cb868b65641,817a9696587da27a
Normal-762.png,Normal,Normal/images/Normal-762.png,Normal/masks/Normal-762.png,True,G16962,train,7eeaf42b55dde7d8bac08e3f3ef9c9bc,99a9c63af8697858
Normal-7620.png,Normal,Normal/images/Normal-7620.png,Normal/masks/Normal-7620.png,True,G16963,train,2e87e84808c1ec38f4ad496f0c4171a5,987d821f494d3f1c
Normal-7621.png,Normal,Normal/images/Normal-7621.png,Normal/masks/Normal-7621.png,True,G16964,train,33a20ec6aaeba111d1e3edc7ccf1e069,cc0c4613534f5f4f
Normal-7622.png,Normal,Normal/images/Normal-7622.png,Normal/masks/Normal-7622.png,True,G16965,train,eec2f495d9a52e37a9203de7f600aef8,dd28d2575d480b5d
Normal-7623.png,Normal,Normal/images/Normal-7623.png,Normal/masks/Normal-7623.png,True,G16966,train,93d7c354e9fc084e1cbb5a03ef7abcd6,9969061e545f794b
Normal-7624.png,Normal,Normal/images/Normal-7624.png,Normal/masks/Normal-7624.png,True,G16967,val,91191a03125c86cab13597208eb56a27,d43cd341abd2435b
Normal-7625.png,Normal,Normal/images/Normal-7625.png,Normal/masks/Normal-7625.png,True,G16968,train,cfabd6c73e15200ab111c8be6d4c2180,d93d821d166f1c4b
Normal-7626.png,Normal,Normal/images/Normal-7626.png,Normal/masks/Normal-7626.png,True,G16969,train,ce71572d96b76a1438739f6763d7316c,c07d421f182d7f2d
Normal-7627.png,Normal,Normal/images/Normal-7627.png,Normal/masks/Normal-7627.png,True,G16970,val,6691ccd581d7ead8facc81c50c416846,dd2ec20b4a4d3cec
Normal-7628.png,Normal,Normal/images/Normal-7628.png,Normal/masks/Normal-7628.png,True,G16971,test,75dfee9d2d705e47148fd75b939d3fda,897847921b458f5f
Normal-7629.png,Normal,Normal/images/Normal-7629.png,Normal/masks/Normal-7629.png,True,G16972,train,0c88633b0b746e0ea38e88f030febca0,d939d4031fcc4b39
Normal-763.png,Normal,Normal/images/Normal-763.png,Normal/masks/Normal-763.png,True,G16973,train,e50836cc01470650216f5219535af62f,907ec73a8d2d38e4
Normal-7630.png,Normal,Normal/images/Normal-7630.png,Normal/masks/Normal-7630.png,True,G16974,train,77e66ad3429cc706fe2f4ef4a2064eed,d91d871e5d4d1c45
Normal-7631.png,Normal,Normal/images/Normal-7631.png,Normal/masks/Normal-7631.png,True,G16975,test,34615693b2c325a70b551d2cb885eefb,cc3dc2431d4f4f0d
Normal-7632.png,Normal,Normal/images/Normal-7632.png,Normal/masks/Normal-7632.png,True,G16976,train,3e403d2115bde0edbd5d57bfbde9b843,cc2a430f6c7eeca2
Normal-7633.png,Normal,Normal/images/Normal-7633.png,Normal/masks/Normal-7633.png,True,G16977,val,75b7ea17d0ae1e3209b0c30012cdc89d,dc7cc3494c0e3f0d
Normal-7634.png,Normal,Normal/images/Normal-7634.png,Normal/masks/Normal-7634.png,True,G16978,train,3583c5f8ce3ba08e1b22b3fe7be6cbd7,ec0cc34b2ceea8bc
Normal-7635.png,Normal,Normal/images/Normal-7635.png,Normal/masks/Normal-7635.png,True,G16979,val,5afd6f24df840c229f1646557f2982dd,8169d61a1a1f5f39
Normal-7636.png,Normal,Normal/images/Normal-7636.png,Normal/masks/Normal-7636.png,True,G16980,test,82bef94c6acb0ea7291f4848a8d956f3,cc3d830b1b475d59
Normal-7637.png,Normal,Normal/images/Normal-7637.png,Normal/masks/Normal-7637.png,True,G16981,test,8369d9e06a9b562db3af553d56b0c41c,c434cb0f198b2dbd
Normal-7638.png,Normal,Normal/images/Normal-7638.png,Normal/masks/Normal-7638.png,True,G16982,val,d7c39160a325f116f14da65c5134180e,9d2c53035f4c5e5a
Normal-7639.png,Normal,Normal/images/Normal-7639.png,Normal/masks/Normal-7639.png,True,G16983,train,51f4ee15c1536f283cc9ef32f65d026b,cc3cd24b5f460d5c
Normal-764.png,Normal,Normal/images/Normal-764.png,Normal/masks/Normal-764.png,True,G16984,train,69dd3d12308863bb157c8777f478ddf7,902bc21edcf03db3
Normal-7640.png,Normal,Normal/images/Normal-7640.png,Normal/masks/Normal-7640.png,True,G16985,train,5851f8633e6e3dfc3ccaffea99e6e70c,816856861a5bed7b
Normal-7641.png,Normal,Normal/images/Normal-7641.png,Normal/masks/Normal-7641.png,True,G16986,train,962846e9a84fa8493de9c13f130b7b69,f979861a184d5b39
Normal-7642.png,Normal,Normal/images/Normal-7642.png,Normal/masks/Normal-7642.png,True,G16987,train,e01e57a1b7ed2aa946653beeb9017156,d9bc869251c77a91
Normal-7643.png,Normal,Normal/images/Normal-7643.png,Normal/masks/Normal-7643.png,True,G16988,train,57b5dfe3258772661afa6b6013a4c895,807b051e1f595d5b
Normal-7644.png,Normal,Normal/images/Normal-7644.png,Normal/masks/Normal-7644.png,True,G16989,train,2de619f3b056b0503773f10b26cf067e,dd2d12471e4a4f39
Normal-7645.png,Normal,Normal/images/Normal-7645.png,Normal/masks/Normal-7645.png,True,G16990,train,9c644fc476c31d5cb038e57aa69fc299,d92d861e1d5f1945
Normal-7646.png,Normal,Normal/images/Normal-7646.png,Normal/masks/Normal-7646.png,True,G16991,train,1277549bd72e6cb063fe6d902952c141,cc3dc35e0b4a3c4d
Normal-7647.png,Normal,Normal/images/Normal-7647.png,Normal/masks/Normal-7647.png,True,G16992,train,5cfd6f82cb05c274d28127f40f6c1212,897bd61a194b3f90
Normal-7648.png,Normal,Normal/images/Normal-7648.png,Normal/masks/Normal-7648.png,True,G16993,train,0644e2ccdd9f1b2c58dd474fdd7beb45,d83b520f0f0c3d3d
Normal-7649.png,Normal,Normal/images/Normal-7649.png,Normal/masks/Normal-7649.png,True,G16994,train,74a45ee4698205442e6168aa5a4c161f,d929d61f5c421f1c
Normal-765.png,Normal,Normal/images/Normal-765.png,Normal/masks/Normal-765.png,True,G16995,test,6b6977e56ba420ceb1ebf8736456e2d9,dc6f830b8d0d6c47
Normal-7650.png,Normal,Normal/images/Normal-7650.png,Normal/masks/Normal-7650.png,True,G16996,val,03066e85844327a7efcd72cae3620d32,8d7cd0c21bd5425f
Normal-7651.png,Normal,Normal/images/Normal-7651.png,Normal/masks/Normal-7651.png,True,G16997,test,e12276c1341eaa3ae7dd05951aa9b7d7,d939465e194d1b3c
Normal-7652.png,Normal,Normal/images/Normal-7652.png,Normal/masks/Normal-7652.png,True,G16998,train,6f2e993a56c43dc35f15cc1850d3fae8,dd3c871b1ac64a99
Normal-7653.png,Normal,Normal/images/Normal-7653.png,Normal/masks/Normal-7653.png,True,G16999,train,c98351b921a794a1c62998e4948adf83,9d6807569b033d1f
Normal-7654.png,Normal,Normal/images/Normal-7654.png,Normal/masks/Normal-7654.png,True,G17000,train,76bb302e3a34aba4dfca5c960ca0c5cd,9939461b194f5f19
Normal-7655.png,Normal,Normal/images/Normal-7655.png,Normal/masks/Normal-7655.png,True,G17001,train,bf85f77f4a00c3015c0053986420ac7b,dc6f430b4f0c0caf
Normal-7656.png,Normal,Normal/images/Normal-7656.png,Normal/masks/Normal-7656.png,True,G17002,train,c07cc96fedd5c00aa0c864010f473c86,d13c064b1b8cecbb
Normal-7657.png,Normal,Normal/images/Normal-7657.png,Normal/masks/Normal-7657.png,True,G17003,train,b7d752072c5ccd0c51a0d524ecec3102,897f07561a4d6a39
Normal-7658.png,Normal,Normal/images/Normal-7658.png,Normal/masks/Normal-7658.png,True,G17004,train,722f886f18206ac7c01e54069e752b5a,a82f821e4c4b7d6b
Normal-7659.png,Normal,Normal/images/Normal-7659.png,Normal/masks/Normal-7659.png,True,G17005,train,f159b5d19cfcbd5abee28cc11035da58,dc2ec24b192f6a3c
Normal-766.png,Normal,Normal/images/Normal-766.png,Normal/masks/Normal-766.png,True,G17006,train,79082358ed848b392a00786cc7403893,912dc63ab86939e9
Normal-7660.png,Normal,Normal/images/Normal-7660.png,Normal/masks/Normal-7660.png,True,G17007,train,83b67bcb197e0b15fe4457cccea946df,81ebd71a80477ae5
Normal-7661.png,Normal,Normal/images/Normal-7661.png,Normal/masks/Normal-7661.png,True,G17008,train,43b858f84bfd96091ccaac88631a15c3,81785f87121e1f7a
Normal-7662.png,Normal,Normal/images/Normal-7662.png,Normal/masks/Normal-7662.png,True,G17009,train,cd925b5ddb7fc2a60ea486bb7afb8cef,d434cb4b4b674b4a
Normal-7663.png,Normal,Normal/images/Normal-7663.png,Normal/masks/Normal-7663.png,True,G17010,train,aa732ddb3bf13333af1988a650df2be5,997dc6521bc10f3c
Normal-7664.png,Normal,Normal/images/Normal-7664.png,Normal/masks/Normal-7664.png,True,G17011,train,e48c580ec93e413e3a25c36f9f15ea5f,9139c64e1b4f4f38
Normal-7665.png,Normal,Normal/images/Normal-7665.png,Normal/masks/Normal-7665.png,True,G17012,val,9ea9e1be822ff43f289604e86b3c9a6c,913e424b1cbce5b9
Normal-7666.png,Normal,Normal/images/Normal-7666.png,Normal/masks/Normal-7666.png,True,G17013,val,5a4629dcee5b501032e00507a234cb35,d96d471e154e194d
Normal-7667.png,Normal,Normal/images/Normal-7667.png,Normal/masks/Normal-7667.png,True,G17014,val,19527aa377931c469c359fc10f56c762,997b0618796d38f1
Normal-7668.png,Normal,Normal/images/Normal-7668.png,Normal/masks/Normal-7668.png,True,G17015,test,15ff006dd2adbe8971bff012e4402bdb,d92f464e1b4d4a3c
Normal-7669.png,Normal,Normal/images/Normal-7669.png,Normal/masks/Normal-7669.png,True,G17016,test,7aea4f91301a68a7ddc2b6e98bb917bd,dd3dc25b19ca2a38
Normal-767.png,Normal,Normal/images/Normal-767.png,Normal/masks/Normal-767.png,True,G17017,train,1fb1031e66614be52d79c37f7e324e02,cd6e830bc54d6cc5
Normal-7670.png,Normal,Normal/images/Normal-7670.png,Normal/masks/Normal-7670.png,True,G17018,val,63b3e3c6090f93a2b702a79e5bb81bf7,d42fc31d0ca19f87
Normal-7671.png,Normal,Normal/images/Normal-7671.png,Normal/masks/Normal-7671.png,True,G17019,test,836f06e7c103930ea77a6bb33cdb4709,d93d86023b52ee5a
Normal-7672.png,Normal,Normal/images/Normal-7672.png,Normal/masks/Normal-7672.png,True,G17020,train,cc42697ea73ac83f8e7b5e282764da58,c93f461e5c0d594d
Normal-7673.png,Normal,Normal/images/Normal-7673.png,Normal/masks/Normal-7673.png,True,G17021,test,c7208c008a05ab6236a008fa956af870,d42d4b1f4f0e4b4a
Normal-7674.png,Normal,Normal/images/Normal-7674.png,Normal/masks/Normal-7674.png,True,G17022,train,8f481678b114ef8b5285da43bbefa7cb,817cc6121bcf3a3b
Normal-7675.png,Normal,Normal/images/Normal-7675.png,Normal/masks/Normal-7675.png,True,G17023,test,cbae63a9709ba7e8e7cf37d9a415c3d9,8929871e9a875eb9
Normal-7676.png,Normal,Normal/images/Normal-7676.png,Normal/masks/Normal-7676.png,True,G17024,train,3f3f22db5878420a2e88237908a0f9bb,d52ee2512b55c257
Normal-7677.png,Normal,Normal/images/Normal-7677.png,Normal/masks/Normal-7677.png,True,G17025,train,53bde7d2443e2a6b5b50fab1dee50117,99790706181f577e
Normal-7678.png,Normal,Normal/images/Normal-7678.png,Normal/masks/Normal-7678.png,True,G17026,train,9d14f83942d3faaa99dd7656872f8838,cc2dc24b0cee2bbc
Normal-7679.png,Normal,Normal/images/Normal-7679.png,Normal/masks/Normal-7679.png,True,G17027,train,6f0eb7174e54bf87a9ff8a145734db70,cd2cc20b556e2cf5
Normal-768.png,Normal,Normal/images/Normal-768.png,Normal/masks/Normal-768.png,True,G17028,train,d626050ce44f93ea078ed2b4337c63e3,d02bc20bbc693df8
Normal-7680.png,Normal,Normal/images/Normal-7680.png,Normal/masks/Normal-7680.png,True,G17029,train,cd1e09cfb61347ba41105d1e7cc3472b,d92d821b1bce1eb8
Normal-7681.png,Normal,Normal/images/Normal-7681.png,Normal/masks/Normal-7681.png,True,G17030,train,0bae90964a51f433aab3ff7c89d40d66,807b56161a1a5f5f
Normal-7682.png,Normal,Normal/images/Normal-7682.png,Normal/masks/Normal-7682.png,True,G17031,train,b22e72b65514619eca4e7e353ce5095a,c02d3fda34938b4d
Normal-7683.png,Normal,Normal/images/Normal-7683.png,Normal/masks/Normal-7683.png,True,G17032,train,6dcefe9cf3e55a834fbc01907969acec,d92a560f461f454f
Normal-7684.png,Normal,Normal/images/Normal-7684.png,Normal/masks/Normal-7684.png,True,G17033,train,4db226e01b2e0d0ef30a814f9ea11729,883f075e186deb38
Normal-7685.png,Normal,Normal/images/Normal-7685.png,Normal/masks/Normal-7685.png,True,G17034,val,7db218e4803e96891c6ba9155d701123,d93c461a1d29bab9
Normal-7686.png,Normal,Normal/images/Normal-7686.png,Normal/masks/Normal-7686.png,True,G17035,train,e07076082314c998e166f9e060edff01,897e470612575f1b
Normal-7687.png,Normal,Normal/images/Normal-7687.png,Normal/masks/Normal-7687.png,True,G17036,val,355449a14e5f47949dece5c8e98c0afd,9969061ad97af8e1
Normal-7688.png,Normal,Normal/images/Normal-7688.png,Normal/masks/Normal-7688.png,True,G17037,test,1cfe3f58f829e446ae856afaf5a1a3fa,dd2cc6d31b404f1b
Normal-7689.png,Normal,Normal/images/Normal-7689.png,Normal/masks/Normal-7689.png,True,G17038,train,6671338a52e661328631c019a42b63cc,dd2c971b0d1f1c4c
Normal-769.png,Normal,Normal/images/Normal-769.png,Normal/masks/Normal-769.png,True,G17039,train,dde753602953321805105c60de78c6a5,d42f424f492f2f34
Normal-7690.png,Normal,Normal/images/Normal-7690.png,Normal/masks/Normal-7690.png,True,G17040,val,2044396f674dd1ff16ea1b91cbae7e78,c13d1642197d6bf1
Normal-7691.png,Normal,Normal/images/Normal-7691.png,Normal/masks/Normal-7691.png,True,G17041,train,6f6ae79252b5f0d64c64a1cab332fc9e,d83d425719134f3d
Normal-7692.png,Normal,Normal/images/Normal-7692.png,Normal/masks/Normal-7692.png,True,G17042,train,4c27fd2719106887fb5747dcc15d2a57,8c2d425b1f4b4e1f
Normal-7693.png,Normal,Normal/images/Normal-7693.png,Normal/masks/Normal-7693.png,True,G17043,train,cdb2c1de3d75a25bcce77066a557d77e,817b87861d1d1e1f
Normal-7694.png,Normal,Normal/images/Normal-7694.png,Normal/masks/Normal-7694.png,True,G17044,test,137d7553fe8c89052f494e952a8cb7a3,917f161a5d58196b
Normal-7695.png,Normal,Normal/images/Normal-7695.png,Normal/masks/Normal-7695.png,True,G17045,train,99de6a15898649484904bc137626864f,dc3dc35b0d0a2d1d
Normal-7696.png,Normal,Normal/images/Normal-7696.png,Normal/masks/Normal-7696.png,True,G17046,train,8815f99ccce94cd4bc9f6e723f231997,992d1618857c39ef
Normal-7697.png,Normal,Normal/images/Normal-7697.png,Normal/masks/Normal-7697.png,True,G17047,train,bf865ddc03f1a9c32bab4a4d5cd8d6b3,cc62530d2bbd24be
Normal-7698.png,Normal,Normal/images/Normal-7698.png,Normal/masks/Normal-7698.png,True,G17048,train,4957bdab22b2d6026c7cab2a96e10dc1,91680e1b7979792d
Normal-7699.png,Normal,Normal/images/Normal-7699.png,Normal/masks/Normal-7699.png,True,G17049,train,6a8c000e41995a5932912b36e53be504,9d6f861a0d4e5a4d
Normal-77.png,Normal,Normal/images/Normal-77.png,Normal/masks/Normal-77.png,True,G17050,train,279fc49bd178b4bc109273c1c8c8d5a8,d46cef2f0d4d1c44
Normal-770.png,Normal,Normal/images/Normal-770.png,Normal/masks/Normal-770.png,True,G17051,val,e12cb7f57acf1539ec20d656aad80597,d17fae3a8c090dcc
Normal-7700.png,Normal,Normal/images/Normal-7700.png,Normal/masks/Normal-7700.png,True,G17052,train,113c70fbac08d151be50050b5607bd4c,897d86121b1d595f
Normal-7701.png,Normal,Normal/images/Normal-7701.png,Normal/masks/Normal-7701.png,True,G17053,train,e356a832bbee2868f05bd13eb46b1d2d,992d871616475f1e
Normal-7702.png,Normal,Normal/images/Normal-7702.png,Normal/masks/Normal-7702.png,True,G17054,val,bdc270dd7318be8e0954549ba5748001,fd46530b4c0f0f2b
Normal-7703.png,Normal,Normal/images/Normal-7703.png,Normal/masks/Normal-7703.png,True,G17055,train,f94b9079e1a18022b64deb02e1a3e0b5,817e07071a5d697b
Normal-7704.png,Normal,Normal/images/Normal-7704.png,Normal/masks/Normal-7704.png,True,G17056,test,5993fa858fb9dcf605c7c07be95252e1,9939061a1d4d78fd
Normal-7705.png,Normal,Normal/images/Normal-7705.png,Normal/masks/Normal-7705.png,True,G17057,train,0b42aae28137bef190fafcc48d3b049e,8169871a155f3a5f
Normal-7706.png,Normal,Normal/images/Normal-7706.png,Normal/masks/Normal-7706.png,True,G17058,test,bb1745a263602b2ef3f04b5c9fe99dcb,d42b53575b5d4b08
Normal-7707.png,Normal,Normal/images/Normal-7707.png,Normal/masks/Normal-7707.png,True,G17059,test,5574d6d92b2c62cbaf99e755d6b3c072,99689611175e7adc
Normal-7708.png,Normal,Normal/images/Normal-7708.png,Normal/masks/Normal-7708.png,True,G17060,train,3cc7d07b971863ac089fa98b9fb74643,d9adc21f954f3c40
Normal-7709.png,Normal,Normal/images/Normal-7709.png,Normal/masks/Normal-7709.png,True,G17061,test,af00d38895f350442578ee4f3155c77d,d83985c7151d471f
Normal-771.png,Normal,Normal/images/Normal-771.png,Normal/masks/Normal-771.png,True,G17062,train,4897387d5c6f00109f386670a4dac8b0,d47ee31bc9a528a4
Normal-7710.png,Normal,Normal/images/Normal-7710.png,Normal/masks/Normal-7710.png,True,G17063,train,4bcb9655a1967d5fcec05c52ad4eb646,c139d65e14475f15
Normal-7711.png,Normal,Normal/images/Normal-7711.png,Normal/masks/Normal-7711.png,True,G17064,train,4766b6387e34dfab0fd40547c6c1801a,d91d47461a1f4d4b
Normal-7712.png,Normal,Normal/images/Normal-7712.png,Normal/masks/Normal-7712.png,True,G17065,train,45f674431ca70f24d039b909d7d5fdac,e97d065b1a4d1b1c
Normal-7713.png,Normal,Normal/images/Normal-7713.png,Normal/masks/Normal-7713.png,True,G17066,train,9f8bccd9098a759ed6f54b9683356cf7,997907461b1c4e5f
Normal-7714.png,Normal,Normal/images/Normal-7714.png,Normal/masks/Normal-7714.png,True,G17067,train,5fd7b4b7d27571eeddb7aae838d5c79b,9d2cc659434e39c7
Normal-7715.png,Normal,Normal/images/Normal-7715.png,Normal/masks/Normal-7715.png,True,G17068,train,024a0631620c1fbc2bac80ba4a7e17d2,d92c861bc44af8eb
Normal-7716.png,Normal,Normal/images/Normal-7716.png,Normal/masks/Normal-7716.png,True,G17069,train,4a227f12801d20746bfd4f13edc67c1d,dd6c861a084f7a79
Normal-7717.png,Normal,Normal/images/Normal-7717.png,Normal/masks/Normal-7717.png,True,G17070,train,5d787d85f378c810d7b28980d5a876cd,916916181a6b7fb9
Normal-7718.png,Normal,Normal/images/Normal-7718.png,Normal/masks/Normal-7718.png,True,G17071,train,f5dd9c78bfb5165ea29fd22e3529cb85,dd2c54524b4f4a1f
Normal-7719.png,Normal,Normal/images/Normal-7719.png,Normal/masks/Normal-7719.png,True,G17072,train,56282038c8cc4c769f8bcb120e4c2b62,d1291606197fef31
Normal-772.png,Normal,Normal/images/Normal-772.png,Normal/masks/Normal-772.png,True,G17073,train,1fea3c8c6d5adfa885230f0a9f6eeb76,810ed65ad463b4e7
Normal-7720.png,Normal,Normal/images/Normal-7720.png,Normal/masks/Normal-7720.png,True,G17074,train,b42338bde18b076f71943ab2f26419be,99691616181edf79
Normal-7721.png,Normal,Normal/images/Normal-7721.png,Normal/masks/Normal-7721.png,True,G17075,val,3b32321b9288a28664666176fa6d8622,d93dc61e124e1ace
Normal-7722.png,Normal,Normal/images/Normal-7722.png,Normal/masks/Normal-7722.png,True,G17076,val,94e570e0366589fd1bcd02fbffff9732,9963021e4b3a4f3f
Normal-7723.png,Normal,Normal/images/Normal-7723.png,Normal/masks/Normal-7723.png,True,G17077,val,0d9431ad9071d6f8e5b5e72ca6993252,993cc61b836e398d
Normal-7724.png,Normal,Normal/images/Normal-7724.png,Normal/masks/Normal-7724.png,True,G17078,test,6fa1db6b9ccd6d664a9ac64403692dfe,cc2f430f442d2cef
Normal-7725.png,Normal,Normal/images/Normal-7725.png,Normal/masks/Normal-7725.png,True,G17079,train,e79b3462c68111bc8561c21723c3842d,dd2cc60b086d7a79
Normal-7726.png,Normal,Normal/images/Normal-7726.png,Normal/masks/Normal-7726.png,True,G17080,train,f912342e3658cb0dbd9b83b3c3a9b1a6,997d961a551a390f
Normal-7727.png,Normal,Normal/images/Normal-7727.png,Normal/masks/Normal-7727.png,True,G17081,train,aadf9d20b6718c42ca5ca1af7b1e421c,997b46165a0d4d5b
Normal-7728.png,Normal,Normal/images/Normal-7728.png,Normal/masks/Normal-7728.png,True,G17082,train,a4733c72690a6144db291e1373334057,997f061a1b3aab38
Normal-7729.png,Normal,Normal/images/Normal-7729.png,Normal/masks/Normal-7729.png,True,G17083,train,9acea44f81f373298d5020b52a741fc1,b9698616871f0e1f
Normal-773.png,Normal,Normal/images/Normal-773.png,Normal/masks/Normal-773.png,True,G17084,train,cda788962ec46b63749918e2291bd4e2,d13bd65e5c4b1858
Normal-7730.png,Normal,Normal/images/Normal-7730.png,Normal/masks/Normal-7730.png,True,G17085,train,4b673af6cf2037925c0715cfc14c0468,9c6f035b095d0a5f
Normal-7731.png,Normal,Normal/images/Normal-7731.png,Normal/masks/Normal-7731.png,True,G17086,train,83e04170b9712c20d4256fe1f86ea54f,d93bd20b1f0a2f19
Normal-7732.png,Normal,Normal/images/Normal-7732.png,Normal/masks/Normal-7732.png,True,G17087,train,1394ab1248f1e767c95f1a170eaca1c5,dc2dc31d4f0e1d0b
Normal-7733.png,Normal,Normal/images/Normal-7733.png,Normal/masks/Normal-7733.png,True,G17088,train,b41e900c96179c71ea5b06ba2937ae9f,dc3cc31b504b1bb9
Normal-7734.png,Normal,Normal/images/Normal-7734.png,Normal/masks/Normal-7734.png,True,G17089,train,24850b596f88bc8a0265866b8e3c85d8,9d2cc21b0b453ef9
Normal-7735.png,Normal,Normal/images/Normal-7735.png,Normal/masks/Normal-7735.png,True,G17090,train,c6a5c6729bbb2540b7bc6635cc753d5c,dc2c464b2de4bcd4
Normal-7736.png,Normal,Normal/images/Normal-7736.png,Normal/masks/Normal-7736.png,True,G17091,train,8108a2095a35c9eaa04690857b3d671e,817d47c61a1f4d5a
Normal-7737.png,Normal,Normal/images/Normal-7737.png,Normal/masks/Normal-7737.png,True,G17092,val,7750ee57a50b84c965587dc51ce26f82,996d461ac15b396e
Normal-7738.png,Normal,Normal/images/Normal-7738.png,Normal/masks/Normal-7738.png,True,G17093,train,6eabddee00ce7fef0293badbced2d901,c939c61715164e3f
Normal-7739.png,Normal,Normal/images/Normal-7739.png,Normal/masks/Normal-7739.png,True,G17094,val,2ed89d47aa68ba229adc2e7130b0f568,91281e1f5b597959
Normal-774.png,Normal,Normal/images/Normal-774.png,Normal/masks/Normal-774.png,True,G17095,test,bcdd60291cbdcc052c64db36c5da3d1b,c136cf1e691c489f
Normal-7740.png,Normal,Normal/images/Normal-7740.png,Normal/masks/Normal-7740.png,True,G17096,test,0a38af92b6f6a9857bc1b4628c8a0709,9568871b18e4fd78
Normal-7741.png,Normal,Normal/images/Normal-7741.png,Normal/masks/Normal-7741.png,True,G17097,val,80d074044147e2411ab00dfa5088ad15,897d021b1f4e7879
Normal-7742.png,Normal,Normal/images/Normal-7742.png,Normal/masks/Normal-7742.png,True,G17098,test,a4dee905b98cc951d06ea5a6bf744286,887c82195ded38fc
Normal-7743.png,Normal,Normal/images/Normal-7743.png,Normal/masks/Normal-7743.png,True,G17099,train,1e86d74878f280219ed9c24d6716403a,dd2ec24b0b470f1d
Normal-7744.png,Normal,Normal/images/Normal-7744.png,Normal/masks/Normal-7744.png,True,G17100,train,3d524d97df6ef33a0d81c1f8c172de18,c02c425f554d3dbd
Normal-7745.png,Normal,Normal/images/Normal-7745.png,Normal/masks/Normal-7745.png,True,G17101,train,e1f988859c5c412b1652e39ed04d0ec8,dd2e831f050f0d4f
Normal-7746.png,Normal,Normal/images/Normal-7746.png,Normal/masks/Normal-7746.png,True,G17102,train,d77adec7fe6f9609db83c839fe8393de,897807521b557b79
Normal-7747.png,Normal,Normal/images/Normal-7747.png,Normal/masks/Normal-7747.png,True,G17103,train,49cf94c977caf6d40e4899ee73d789d7,8829c71fd54e1e5a
Normal-7748.png,Normal,Normal/images/Normal-7748.png,Normal/masks/Normal-7748.png,True,G17104,train,f62e32e4c0420715e3f985f5d00bae38,8c6cd2d34b4f491b
Normal-7749.png,Normal,Normal/images/Normal-7749.png,Normal/masks/Normal-7749.png,True,G17105,train,17565641ff3a38cd4112ba9500c35aca,996d061e051e5f5e
Normal-775.png,Normal,Normal/images/Normal-775.png,Normal/masks/Normal-775.png,True,G17106,train,7d7c54f139c800eddc4b37a037317c46,913dc71ad8693d49
Normal-7750.png,Normal,Normal/images/Normal-7750.png,Normal/masks/Normal-7750.png,True,G17107,train,9c13c4799d19ef6ff928e345f53dd0f1,8869c71e174f1a5d
Normal-7751.png,Normal,Normal/images/Normal-7751.png,Normal/masks/Normal-7751.png,True,G17108,train,7d9c308a85c3b4e1dba9ef59158e0e6c,c87f05161a1b5f3a
Normal-7752.png,Normal,Normal/images/Normal-7752.png,Normal/masks/Normal-7752.png,True,G17109,train,637caaa15cca94a3ccf08e5773876a8f,d72ad1410f464e6f
Normal-7753.png,Normal,Normal/images/Normal-7753.png,Normal/masks/Normal-7753.png,True,G17110,train,f471aadd4999de0b90a90551ade1d0d5,993f461b17071f15
Normal-7754.png,Normal,Normal/images/Normal-7754.png,Normal/masks/Normal-7754.png,True,G17111,val,f80ca116ad755719869875399e0a09ef,d92bd6531e544b19
Normal-7755.png,Normal,Normal/images/Normal-7755.png,Normal/masks/Normal-7755.png,True,G17112,test,8450294caebb8f98952d4a604f3639ae,dd2f421f06471f19
Normal-7756.png,Normal,Normal/images/Normal-7756.png,Normal/masks/Normal-7756.png,True,G17113,train,57f5a25204f823212895b7437c115d74,897a871a934f3aa9
Normal-7757.png,Normal,Normal/images/Normal-7757.png,Normal/masks/Normal-7757.png,True,G17114,train,a958340539c6039187e2533909622859,9879c79a13c55e19
Normal-7758.png,Normal,Normal/images/Normal-7758.png,Normal/masks/Normal-7758.png,True,G17115,val,ded7dba05c57814ff6b330e75add25c9,d13d460a3a0faf39
Normal-7759.png,Normal,Normal/images/Normal-7759.png,Normal/masks/Normal-7759.png,True,G17116,val,a8258a06de2193dd065ab3d674669319,fd2cc2c35d0d4c1d
Normal-776.png,Normal,Normal/images/Normal-776.png,Normal/masks/Normal-776.png,True,G17117,test,7198dec89a1b9ded6158d1e6463191b4,916cc71acd5a92c7
Normal-7760.png,Normal,Normal/images/Normal-7760.png,Normal/masks/Normal-7760.png,True,G17118,train,1d495b8cc7fe6927f3aae45768088765,d42c7543331b1f2b
Normal-7761.png,Normal,Normal/images/Normal-7761.png,Normal/masks/Normal-7761.png,True,G17119,test,32a1b4577743e1fbe931bcbeccf4ea0e,dd6cc61987079a8b
Normal-7762.png,Normal,Normal/images/Normal-7762.png,Normal/masks/Normal-7762.png,True,G17120,train,2095571860c522155204d6fbc2c0fd35,9128161b1b3b5b3f
Normal-7763.png,Normal,Normal/images/Normal-7763.png,Normal/masks/Normal-7763.png,True,G17121,train,184a062c18b6b8fc2e95294872fb8519,d928c40f174f1b5d
Normal-7764.png,Normal,Normal/images/Normal-7764.png,Normal/masks/Normal-7764.png,True,G17122,train,1ec916bd8ac5289e77569c3225e12b8d,d52e03530b4cff58
Normal-7765.png,Normal,Normal/images/Normal-7765.png,Normal/masks/Normal-7765.png,True,G17123,train,b9977d90714487ce487fc1eff7398be9,d93d861a96471acd
Normal-7766.png,Normal,Normal/images/Normal-7766.png,Normal/masks/Normal-7766.png,True,G17124,val,77ee781d526decdfc428cc1947249b6c,992c46575b445f39
Normal-7767.png,Normal,Normal/images/Normal-7767.png,Normal/masks/Normal-7767.png,True,G17125,train,0b8e6dc1760e1302ee269e3bcda43618,d93ec61b114f1c5a
Normal-7768.png,Normal,Normal/images/Normal-7768.png,Normal/masks/Normal-7768.png,True,G17126,train,be9f3c6be2dc3585b0011f199f36d4dd,91695e161b5f6938
Normal-7769.png,Normal,Normal/images/Normal-7769.png,Normal/masks/Normal-7769.png,True,G17127,train,38c3bf626724755b5351cb741a9640c6,81398e56b94fad19
Normal-777.png,Normal,Normal/images/Normal-777.png,Normal/masks/Normal-777.png,True,G17128,train,7cdeab141a9f395eb77695d7fec2bac5,d0bae39bbcc45834
Normal-7770.png,Normal,Normal/images/Normal-7770.png,Normal/masks/Normal-7770.png,True,G17129,val,5c9d2d3f2aa4034e8cd00ea26d23ad31,dd2d079f8a471e18
Normal-7771.png,Normal,Normal/images/Normal-7771.png,Normal/masks/Normal-7771.png,True,G17130,val,f4f3e5f39811d6aa8a9ca58f788065ef,f684fbe1e5a4a4a0
Normal-7772.png,Normal,Normal/images/Normal-7772.png,Normal/masks/Normal-7772.png,True,G17131,train,f4f84d41b702f2d3abe39a1981227f26,dd2b52c31e0f4e1a
Normal-7773.png,Normal,Normal/images/Normal-7773.png,Normal/masks/Normal-7773.png,True,G17132,test,c0bc23ca891d5a9c7a4c12ee0e65ae66,dc2c52c30b5d4e1f
Normal-7774.png,Normal,Normal/images/Normal-7774.png,Normal/masks/Normal-7774.png,True,G17133,train,a340827b0fbe1cdfecb818bef3a35c13,cc3dc20b1f6c1cad
Normal-7775.png,Normal,Normal/images/Normal-7775.png,Normal/masks/Normal-7775.png,True,G17134,train,7898983586588b0c7dbfa20d9ffc68de,98ffc71e944e1843
Normal-7776.png,Normal,Normal/images/Normal-7776.png,Normal/masks/Normal-7776.png,True,G17135,train,b7caf87be1ac76241697ace2d9be9346,997f075e194d78a0
Normal-7777.png,Normal,Normal/images/Normal-7777.png,Normal/masks/Normal-7777.png,True,G17136,test,5f2cdeea35699eeba05c770a6503866b,d80f861b0d1a1f5f
Normal-7778.png,Normal,Normal/images/Normal-7778.png,Normal/masks/Normal-7778.png,True,G17137,test,0e1412f50d38c75dbdc002573d30ee50,9939861e1dce1b1d
Normal-7779.png,Normal,Normal/images/Normal-7779.png,Normal/masks/Normal-7779.png,True,G17138,train,e09a17631b148389857d240fd43cdd0e,d17b061e145f1a5e
Normal-778.png,Normal,Normal/images/Normal-778.png,Normal/masks/Normal-778.png,True,G17139,train,b7ac4c5717823b59201ed00bed01f6f5,d1cf871acd2d21c6
Normal-7780.png,Normal,Normal/images/Normal-7780.png,Normal/masks/Normal-7780.png,True,G17140,train,7d9edd375f3d54a57de6349d65830cfc,d83d074a1d4f4e78
Normal-7781.png,Normal,Normal/images/Normal-7781.png,Normal/masks/Normal-7781.png,True,G17141,val,afd0177427dbae67eb61ddf67bbe23d5,d87b441e1e4f1b19
Normal-7782.png,Normal,Normal/images/Normal-7782.png,Normal/masks/Normal-7782.png,True,G17142,train,7be79dfd17bd2c853a164b38d27aef92,a83d828b0f0d1fbd
Normal-7783.png,Normal,Normal/images/Normal-7783.png,Normal/masks/Normal-7783.png,True,G17143,train,94aa9f1a279a47582eefee3ef10bd8fc,cd28d387d24b96e2
Normal-7784.png,Normal,Normal/images/Normal-7784.png,Normal/masks/Normal-7784.png,True,G17144,train,4fe29d3a563843c49c7f78b5e3618a81,c028d65f1f544f1d
Normal-7785.png,Normal,Normal/images/Normal-7785.png,Normal/masks/Normal-7785.png,True,G17145,train,9ef7503cea3bf621a2635f8df6e78a57,983d061e155e1d5f
Normal-7786.png,Normal,Normal/images/Normal-7786.png,Normal/masks/Normal-7786.png,True,G17146,train,ee415d83628097ac8ffd96f3e0b627ce,dd2e425b0b6d2c3c
Normal-7787.png,Normal,Normal/images/Normal-7787.png,Normal/masks/Normal-7787.png,True,G17147,test,07bb4f1d00374d9755af42b24d033a23,dc2f831f0a0f1d4e
Normal-7788.png,Normal,Normal/images/Normal-7788.png,Normal/masks/Normal-7788.png,True,G17148,train,b563d83066342cac9d34e50b333b4e4a,dd2e861b454f18e5
Normal-7789.png,Normal,Normal/images/Normal-7789.png,Normal/masks/Normal-7789.png,True,G17149,train,3605db9fb872d0db8f796f7f6b694a5f,c03dd3531f1a0b2f
Normal-779.png,Normal,Normal/images/Normal-779.png,Normal/masks/Normal-779.png,True,G17150,train,0f207a5889467ac4f01db1866a769fd0,d03dc33fb8693868
Normal-7790.png,Normal,Normal/images/Normal-7790.png,Normal/masks/Normal-7790.png,True,G17151,train,3314780a7786ba334c6f0a07093460a0,99780653581c6bfe
Normal-7791.png,Normal,Normal/images/Normal-7791.png,Normal/masks/Normal-7791.png,True,G17152,train,a9a9bbb69b3c3fd9a03b08ad7f936bd3,cd2ec2031f4c7f38
Normal-7792.png,Normal,Normal/images/Normal-7792.png,Normal/masks/Normal-7792.png,True,G17153,val,d59704f2e4a6e2b5114b0ba1876a1936,cd3cc30b440abdeb
Normal-7793.png,Normal,Normal/images/Normal-7793.png,Normal/masks/Normal-7793.png,True,G17154,train,25b78d414ca66cafe10f5b03ce2de19c,9139165d7be069e1
Normal-7794.png,Normal,Normal/images/Normal-7794.png,Normal/masks/Normal-7794.png,True,G17155,train,9414d21cd93e6ceb58d98234e31228ff,917947121f493a7d
Normal-7795.png,Normal,Normal/images/Normal-7795.png,Normal/masks/Normal-7795.png,True,G17156,train,56e1511d59f298b0bd952dd13f57a7cf,cc3cd24b1d4b2f1c
Normal-7796.png,Normal,Normal/images/Normal-7796.png,Normal/masks/Normal-7796.png,True,G17157,val,c70437c13949db611a64a97e281be61e,997f85061a854f1f
Normal-7797.png,Normal,Normal/images/Normal-7797.png,Normal/masks/Normal-7797.png,True,G17158,val,36d93fe1a3e8dbbeabbafda5beba0eaa,9c7cc203fd40eeb4
Normal-7798.png,Normal,Normal/images/Normal-7798.png,Normal/masks/Normal-7798.png,True,G17159,test,1161f2f0b027df71771c4823c7392ccb,cc3dc3430bd34b0f
Normal-7799.png,Normal,Normal/images/Normal-7799.png,Normal/masks/Normal-7799.png,True,G17160,train,d2c6134c056dd0dd07969197461198cf,dd2cc21b1f461a5d
Normal-78.png,Normal,Normal/images/Normal-78.png,Normal/masks/Normal-78.png,True,G17161,test,4b0280a0fd4b2154a497165b10e55011,956eaf1acd2c4cc4
Normal-780.png,Normal,Normal/images/Normal-780.png,Normal/masks/Normal-780.png,True,G17162,train,c9626ef42a1ff43c3a6333772174f039,d82fe71a29693c4c
Normal-7800.png,Normal,Normal/images/Normal-7800.png,Normal/masks/Normal-7800.png,True,G17163,train,e8274b1bd6525741a3a22fb85a070eb3,dd2cc21b5d4c1e6c
Normal-7801.png,Normal,Normal/images/Normal-7801.png,Normal/masks/Normal-7801.png,True,G17164,train,ed515762cd27fdc548169f96a111c0b0,d92f461b0e0b396b
Normal-7802.png,Normal,Normal/images/Normal-7802.png,Normal/masks/Normal-7802.png,True,G17165,train,c67dfaf250e9dcbc9fdb3322921df0eb,8d2f55521bd74a1a
Normal-7803.png,Normal,Normal/images/Normal-7803.png,Normal/masks/Normal-7803.png,True,G17166,train,72ed1ea73c12e299e637d20548b0b8de,d829871e154f1c7d
Normal-7804.png,Normal,Normal/images/Normal-7804.png,Normal/masks/Normal-7804.png,True,G17167,train,32eec868666f5a2818bba83ace7ce7d4,9179461ed05f5c5a
Normal-7805.png,Normal,Normal/images/Normal-7805.png,Normal/masks/Normal-7805.png,True,G17168,val,e100262c0190a5272c8a139b771c44c9,cd3c420f444ffc6a
Normal-7806.png,Normal,Normal/images/Normal-7806.png,Normal/masks/Normal-7806.png,True,G17169,train,989d6f1ac7e3f57b3294528f3299c6a7,b879879696871e1a
Normal-7807.png,Normal,Normal/images/Normal-7807.png,Normal/masks/Normal-7807.png,True,G17170,val,bad05ade99585e4dc5bd4f144dfa377a,9c6c434f4c4e6c4f
Normal-7808.png,Normal,Normal/images/Normal-7808.png,Normal/masks/Normal-7808.png,True,G17171,train,02f0ad94b0b3a8b466b1c3792c3a33dd,d939451e1c4b3d5a
Normal-7809.png,Normal,Normal/images/Normal-7809.png,Normal/masks/Normal-7809.png,True,G17172,test,75d057cb4dd27068edbe268c7be5c72c,c9f9d70ad44d5941
Normal-781.png,Normal,Normal/images/Normal-781.png,Normal/masks/Normal-781.png,True,G17173,train,4ae8f02718b3025e31522d20df49f2ca,d43ea73a8d2c3c4c
Normal-7810.png,Normal,Normal/images/Normal-7810.png,Normal/masks/Normal-7810.png,True,G17174,train,bf9ef9d19ca467365934ae0dfcd558b1,d97cc217471f890e
Normal-7811.png,Normal,Normal/images/Normal-7811.png,Normal/masks/Normal-7811.png,True,G17175,train,6b7a1d9de7d899d601ad7954fb321e2a,cd3db0c2421fd3c3
Normal-7812.png,Normal,Normal/images/Normal-7812.png,Normal/masks/Normal-7812.png,True,G17176,train,1c77a26bfffa108f27d8f7a277ac9730,917857161a1f3e39
Normal-7813.png,Normal,Normal/images/Normal-7813.png,Normal/masks/Normal-7813.png,True,G17177,train,35d9f89925f010939b5b20b7d7f3904f,887b560a5a595b5d
Normal-7814.png,Normal,Normal/images/Normal-7814.png,Normal/masks/Normal-7814.png,True,G17178,val,2eefb4f2d25c7893babdcb49ad1ec25d,dc07834b0c475b5f
Normal-7815.png,Normal,Normal/images/Normal-7815.png,Normal/masks/Normal-7815.png,True,G17179,test,221df303920e8152fa4283cad81bffc5,dd2dc30b4d1a4b4b
Normal-7816.png,Normal,Normal/images/Normal-7816.png,Normal/masks/Normal-7816.png,True,G17180,test,4e085fb4508b41c543c2e55de70de4cc,997b07871e174517
Normal-7817.png,Normal,Normal/images/Normal-7817.png,Normal/masks/Normal-7817.png,True,G17181,train,116371260012eb50503d837ed3d39abb,d929c6161ad54eba
Normal-7818.png,Normal,Normal/images/Normal-7818.png,Normal/masks/Normal-7818.png,True,G17182,train,bd741b2e5924b602f31894f08607db15,d9a8d74f544a1d54
Normal-7819.png,Normal,Normal/images/Normal-7819.png,Normal/masks/Normal-7819.png,True,G17183,train,fb74ec41ceff57138b2dddc7ae09854e,997d871a1d4c5a4d
Normal-782.png,Normal,Normal/images/Normal-782.png,Normal/masks/Normal-782.png,True,G17184,train,c8ddedd3f0ac1b493916dc8a6ea40a16,902bd27ad87919d3
Normal-7820.png,Normal,Normal/images/Normal-7820.png,Normal/masks/Normal-7820.png,True,G17185,train,133df4c3cf36906b06e3f91007a51310,c46c2f4b39b664b4
Normal-7821.png,Normal,Normal/images/Normal-7821.png,Normal/masks/Normal-7821.png,True,G17186,train,17d8f0643bd8e658e5d39a0b0af8f0dd,d43d43430b0f4f1f
Normal-7822.png,Normal,Normal/images/Normal-7822.png,Normal/masks/Normal-7822.png,True,G17187,train,0328f6837f8a7373e51b43ad23e12763,987b05861a58e9fd
Normal-7823.png,Normal,Normal/images/Normal-7823.png,Normal/masks/Normal-7823.png,True,G17188,train,c661acc399676c407b170e5a508bffb8,dd2e424b1a4c2fbc
Normal-7824.png,Normal,Normal/images/Normal-7824.png,Normal/masks/Normal-7824.png,True,G17189,train,52ace48ff3969a495fde73e59464008e,d92b05161f465f5c
Normal-7825.png,Normal,Normal/images/Normal-7825.png,Normal/masks/Normal-7825.png,True,G17190,train,cc0e8d7bb67535c2294dc0387f12c49d,dd680646115f5e79
Normal-7826.png,Normal,Normal/images/Normal-7826.png,Normal/masks/Normal-7826.png,True,G17191,train,86a611103f4121f89fe64696c336f674,dc2ed2430f4e2f0d
Normal-7827.png,Normal,Normal/images/Normal-7827.png,Normal/masks/Normal-7827.png,True,G17192,train,aeead9ff1148e4d3c5df15858e763029,937b941e1b5a3a19
Normal-7828.png,Normal,Normal/images/Normal-7828.png,Normal/masks/Normal-7828.png,True,G17193,train,0737545afb1c55f6ba770c575c211faf,8979871711871eb7
Normal-7829.png,Normal,Normal/images/Normal-7829.png,Normal/masks/Normal-7829.png,True,G17194,train,f853ce88505d7fedddd886886871d76f,817f5e18593a5656
Normal-783.png,Normal,Normal/images/Normal-783.png,Normal/masks/Normal-783.png,True,G17195,val,227ee464b28680d54edc98e3ecef934d,903bc63ab8599dc6
Normal-7830.png,Normal,Normal/images/Normal-7830.png,Normal/masks/Normal-7830.png,True,G17196,val,0330eeb16d69729deac55ef8959dae07,d9f9471a565a10ce
Normal-7831.png,Normal,Normal/images/Normal-7831.png,Normal/masks/Normal-7831.png,True,G17197,val,ecd4aab801b1b0491e56164507041534,d13d5a86183df139
Normal-7832.png,Normal,Normal/images/Normal-7832.png,Normal/masks/Normal-7832.png,True,G17198,test,802758066b6b97946d51ab38ca6d93c6,997b061e17581bce
Normal-7833.png,Normal,Normal/images/Normal-7833.png,Normal/masks/Normal-7833.png,True,G17199,train,bb5d0ef8318217b7c69df6677a486c99,dd2dc24f1a4e5d18
Normal-7834.png,Normal,Normal/images/Normal-7834.png,Normal/masks/Normal-7834.png,True,G17200,train,22d15ce1c60db029f65ef30a1fc19d17,9978c7125a4f5a5a
Normal-7835.png,Normal,Normal/images/Normal-7835.png,Normal/masks/Normal-7835.png,True,G17201,test,92ac5db5245fab1d934e2faa6fb398a5,cd2ec25b4d4c1cec
Normal-7836.png,Normal,Normal/images/Normal-7836.png,Normal/masks/Normal-7836.png,True,G17202,test,fc5b242e90280a51f15afa418ac161e2,d92f465b144b1d4e
Normal-7837.png,Normal,Normal/images/Normal-7837.png,Normal/masks/Normal-7837.png,True,G17203,train,aa504fd65cd39e3cb3033ae9c9399df4,883b161e5d2979b9
Normal-7838.png,Normal,Normal/images/Normal-7838.png,Normal/masks/Normal-7838.png,True,G17204,train,e2c341ba45df5c997c62c81665c50c7e,9968c612055e7b5f
Normal-7839.png,Normal,Normal/images/Normal-7839.png,Normal/masks/Normal-7839.png,True,G17205,test,c5b6c9fd32b2342566fa6e0463ca9b30,c17bd6c259904f99
Normal-784.png,Normal,Normal/images/Normal-784.png,Normal/masks/Normal-784.png,True,G17206,train,fd3d6546d43bbc53756d7cdff78bb719,c46f475f4d4c0c87
Normal-7840.png,Normal,Normal/images/Normal-7840.png,Normal/masks/Normal-7840.png,True,G17207,train,05854edfc438daed64cab72406fd6878,913ec619465d1d4f
Normal-7841.png,Normal,Normal/images/Normal-7841.png,Normal/masks/Normal-7841.png,True,G17208,train,e871697ed4706630dcaf3b6228679f04,917906821b3ded3b
Normal-7842.png,Normal,Normal/images/Normal-7842.png,Normal/masks/Normal-7842.png,True,G17209,train,027ad595a55d855b06def0f8b9bfad12,917e86531b5c4e59
Normal-7843.png,Normal,Normal/images/Normal-7843.png,Normal/masks/Normal-7843.png,True,G17210,train,93b92e7b7322ade0262181a06dc1e630,8179e7d65e15441e
Normal-7844.png,Normal,Normal/images/Normal-7844.png,Normal/masks/Normal-7844.png,True,G17211,train,13211a8d67925d722b9642ba85c21b11,997f8716194d2e1c
Normal-7845.png,Normal,Normal/images/Normal-7845.png,Normal/masks/Normal-7845.png,True,G17212,train,68333e295e2ce2174c4d37ed0137a92f,dc54530b4f2a2f2e
Normal-7846.png,Normal,Normal/images/Normal-7846.png,Normal/masks/Normal-7846.png,True,G17213,train,accb5f502fa3b58af924e467232f536a,d939061ff868b9e0
Normal-7847.png,Normal,Normal/images/Normal-7847.png,Normal/masks/Normal-7847.png,True,G17214,train,03ad33f0f724da025966cc9cf0262aff,887b87941a4f7a5a
Normal-7848.png,Normal,Normal/images/Normal-7848.png,Normal/masks/Normal-7848.png,True,G17215,train,97544ad6db727eec792a0ca74307fd06,c03dd7534a1e4f1a
Normal-7849.png,Normal,Normal/images/Normal-7849.png,Normal/masks/Normal-7849.png,True,G17216,val,0b82118b096ec17aeb42eafa0a2fb501,d929c6df1084ef19
Normal-785.png,Normal,Normal/images/Normal-785.png,Normal/masks/Normal-785.png,True,G17217,val,aa2daa4924ae873e1a58564d935ffe62,993bd21a584939db
Normal-7850.png,Normal,Normal/images/Normal-7850.png,Normal/masks/Normal-7850.png,True,G17218,val,42bf9a6d82015d3fa115821bc4a132f6,d179561b190f393c
Normal-7851.png,Normal,Normal/images/Normal-7851.png,Normal/masks/Normal-7851.png,True,G17219,val,f8beebb10033ae0e136aae87ff4427b5,dd7d42521b462f38
Normal-7852.png,Normal,Normal/images/Normal-7852.png,Normal/masks/Normal-7852.png,True,G17220,test,d307b565deeffb24c3b10bf0cd53bfd0,d96ac4131a4b1f5b
Normal-7853.png,Normal,Normal/images/Normal-7853.png,Normal/masks/Normal-7853.png,True,G17221,test,95b6ece1e8a8527a51ed7f97c0dd22b7,9928071e5cec78f9
Normal-7854.png,Normal,Normal/images/Normal-7854.png,Normal/masks/Normal-7854.png,True,G17222,train,bfe823dc2bab3f93c864d1e9c428758e,817b56861e19f9c3
Normal-7855.png,Normal,Normal/images/Normal-7855.png,Normal/masks/Normal-7855.png,True,G17223,train,0fb89638d58ec10f68d652ee3cd200e2,d43cc7d31d4e382c
Normal-7856.png,Normal,Normal/images/Normal-7856.png,Normal/masks/Normal-7856.png,True,G17224,train,052fa9ad989b54a5c1eab82e2d528dad,dd2c461ec44ef8c3
Normal-7857.png,Normal,Normal/images/Normal-7857.png,Normal/masks/Normal-7857.png,True,G17225,train,4aefbcff65c15858546683396a774cb4,987f061e1f0d1a5d
Normal-7858.png,Normal,Normal/images/Normal-7858.png,Normal/masks/Normal-7858.png,True,G17226,test,18082b2bc896fa196a5dde806632c75d,9d7c861b454f3945
Normal-7859.png,Normal,Normal/images/Normal-7859.png,Normal/masks/Normal-7859.png,True,G17227,train,d1bda6a3d5bf8ca75ef044cb2e7231e7,d90946c63b2de4b5
Normal-786.png,Normal,Normal/images/Normal-786.png,Normal/masks/Normal-786.png,True,G17228,test,e6832fb786717e275863e5615b76257a,997e861a955d9496
Normal-7860.png,Normal,Normal/images/Normal-7860.png,Normal/masks/Normal-7860.png,True,G17229,train,6610b9a1df3cd6b3054e1cfe055a41fc,d929c50e1e457f1a
Normal-7861.png,Normal,Normal/images/Normal-7861.png,Normal/masks/Normal-7861.png,True,G17230,train,6b46b9e66562bbe3588fcfb835387c11,c1853eca3b3e862e
Normal-7862.png,Normal,Normal/images/Normal-7862.png,Normal/masks/Normal-7862.png,True,G17231,train,44498c296bf2a94756f86329a9c5251a,dd2c07190f6c3c6e
Normal-7863.png,Normal,Normal/images/Normal-7863.png,Normal/masks/Normal-7863.png,True,G17232,train,87fefcab2d8401edd65164a7b4b43bd9,d92dc64a1f0c6b39
Normal-7864.png,Normal,Normal/images/Normal-7864.png,Normal/masks/Normal-7864.png,True,G17233,train,855c5ea84f6f6c3db3d32a68caa69778,953a43175a475b1b
Normal-7865.png,Normal,Normal/images/Normal-7865.png,Normal/masks/Normal-7865.png,True,G17234,train,3d1b2df9c9b37b47220610f31b4a8801,c939035f1e4a3db8
Normal-7866.png,Normal,Normal/images/Normal-7866.png,Normal/masks/Normal-7866.png,True,G17235,train,8dcdcda288306cc51f69819c641f5873,897f04165f095d79
Normal-7867.png,Normal,Normal/images/Normal-7867.png,Normal/masks/Normal-7867.png,True,G17236,train,a970bae4a2be7e7395d0b1c9b5aac792,a83985c7161e4f1f
Normal-7868.png,Normal,Normal/images/Normal-7868.png,Normal/masks/Normal-7868.png,True,G17237,train,d08c6281b3ac075d60c98a0e8bba496d,d5bdc25753461a1a
Normal-7869.png,Normal,Normal/images/Normal-7869.png,Normal/masks/Normal-7869.png,True,G17238,train,19260f163c3c6a47da1f7a641a951271,dc2ec35f604f3d40
Normal-787.png,Normal,Normal/images/Normal-787.png,Normal/masks/Normal-787.png,True,G17239,val,7f5bc055ce89a0e1960d1f3acdbd2269,9d7d070a096c78ed
Normal-7870.png,Normal,Normal/images/Normal-7870.png,Normal/masks/Normal-7870.png,True,G17240,val,ad7631426ad0676d18f27afeca37d950,913b861a974e195f
Normal-7871.png,Normal,Normal/images/Normal-7871.png,Normal/masks/Normal-7871.png,True,G17241,test,98033734debe96ea281a671d3ebce169,dd2ac21f820ab8ef
Normal-7872.png,Normal,Normal/images/Normal-7872.png,Normal/masks/Normal-7872.png,True,G17242,train,024ca1398dfc513b04cd3f4fb6ded3b0,897d061b1b7d38a9
Normal-7873.png,Normal,Normal/images/Normal-7873.png,Normal/masks/Normal-7873.png,True,G17243,train,6f248371f23cea9524be6678ee0c24c5,c42cfb4b9c5a2c5c
Normal-7874.png,Normal,Normal/images/Normal-7874.png,Normal/masks/Normal-7874.png,True,G17244,train,b1c32c9edc9a36f339b17531a4a42bf1,c039420b3bb5abb7
Normal-7875.png,Normal,Normal/images/Normal-7875.png,Normal/masks/Normal-7875.png,True,G17245,train,6de3bc2777160f0b918e3b6132a5c255,d53d424b0b0bef0b
Normal-7876.png,Normal,Normal/images/Normal-7876.png,Normal/masks/Normal-7876.png,True,G17246,train,6d2f7eeb163cc308bc8b38fe146512db,a96d061e1c5f1d59
Normal-7877.png,Normal,Normal/images/Normal-7877.png,Normal/masks/Normal-7877.png,True,G17247,train,e749ce8119da6784ea8001970ce5616d,817856863a1f69be
Normal-7878.png,Normal,Normal/images/Normal-7878.png,Normal/masks/Normal-7878.png,True,G17248,train,ab2a1d64bea49fbb78a34e3b075abaa2,d93b46471f00ee1b
Normal-7879.png,Normal,Normal/images/Normal-7879.png,Normal/masks/Normal-7879.png,True,G17249,train,753b16e618f8bbedffef6a959056576f,993f861a59057b55
Normal-788.png,Normal,Normal/images/Normal-788.png,Normal/masks/Normal-788.png,True,G17250,val,5e7b767948980456f93bf29361ebbeb0,d02bc31becc83dd1
Normal-7880.png,Normal,Normal/images/Normal-7880.png,Normal/masks/Normal-7880.png,True,G17251,train,11639b40100a4057e4db34b19e78012c,81795e873a956d91
Normal-7881.png,Normal,Normal/images/Normal-7881.png,Normal/masks/Normal-7881.png,True,G17252,val,1eebf47b7233493ba5f3df1e3a042fb9,d43d0b0b1b4d4e7c
Normal-7882.png,Normal,Normal/images/Normal-7882.png,Normal/masks/Normal-7882.png,True,G17253,train,24ee4b126ffe2b50dafcdd18814d0100,d978c75a874e1949
Normal-7883.png,Normal,Normal/images/Normal-7883.png,Normal/masks/Normal-7883.png,True,G17254,val,5a3e99f25e0134139dcc99747ed95eb9,887dc71a5b451c79
Normal-7884.png,Normal,Normal/images/Normal-7884.png,Normal/masks/Normal-7884.png,True,G17255,test,760f9f351742f21b1244c63d2b48ab21,c43cbb53ab0aab43
Normal-7885.png,Normal,Normal/images/Normal-7885.png,Normal/masks/Normal-7885.png,True,G17256,test,b6e2a5327251b69d277f8d9396fc0981,dc2cc243bd42ed1d
Normal-7886.png,Normal,Normal/images/Normal-7886.png,Normal/masks/Normal-7886.png,True,G17257,train,2782a1dac58b8fa159acd4ef36b1835a,dc3dc35b40460fce
Normal-7887.png,Normal,Normal/images/Normal-7887.png,Normal/masks/Normal-7887.png,True,G17258,train,b0075c0f7cd47ef7ff19093f33bcbce9,8c4cc2434f6dae6d
Normal-7888.png,Normal,Normal/images/Normal-7888.png,Normal/masks/Normal-7888.png,True,G17259,test,654c49cb78c386928000e944552b1c1a,cd3c530b554a1f4d
Normal-7889.png,Normal,Normal/images/Normal-7889.png,Normal/masks/Normal-7889.png,True,G17260,val,5e4d02650f1e250b12e0235608a7b958,887f021b5f0d3d2d
Normal-789.png,Normal,Normal/images/Normal-789.png,Normal/masks/Normal-789.png,True,G17261,train,babe58ccbcb4ef974eb97125c90e439e,d42ae33bc70b3c64
Normal-7890.png,Normal,Normal/images/Normal-7890.png,Normal/masks/Normal-7890.png,True,G17262,train,88f2b1f35a39ea2f3216a35b59947e74,d82b851e1f4e1e1d
Normal-7891.png,Normal,Normal/images/Normal-7891.png,Normal/masks/Normal-7891.png,True,G17263,train,5389e12063132913fda7c5100964c1bc,9921ee1ed116ce5a
Normal-7892.png,Normal,Normal/images/Normal-7892.png,Normal/masks/Normal-7892.png,True,G17264,train,c72a822e497d090b58381e33d420e453,dc3d420b4f0c1f3d
Normal-7893.png,Normal,Normal/images/Normal-7893.png,Normal/masks/Normal-7893.png,True,G17265,train,cc49b433ca9424669dfd916bfaa42d31,9c2d071a431f1f4f
Normal-7894.png,Normal,Normal/images/Normal-7894.png,Normal/masks/Normal-7894.png,True,G17266,train,c19e9c65979af1aa6e9756bfbca26e13,991d461b1e4f1a1f
Normal-7895.png,Normal,Normal/images/Normal-7895.png,Normal/masks/Normal-7895.png,True,G17267,train,bdb7cb8edb541526bc7f5c7ade239e86,817f071819797ae9
Normal-7896.png,Normal,Normal/images/Normal-7896.png,Normal/masks/Normal-7896.png,True,G17268,train,5ae8f049e32d530b4ba6ee23d9b37eee,93788c1c5b5a5b78
Normal-7897.png,Normal,Normal/images/Normal-7897.png,Normal/masks/Normal-7897.png,True,G17269,train,e8df393107e2529813e9f5b3debec0f3,b91987161e0d5f1d
Normal-7898.png,Normal,Normal/images/Normal-7898.png,Normal/masks/Normal-7898.png,True,G17270,train,61e3e616c34b4404d9d779488b40bce9,d17f06061a1f073f
Normal-7899.png,Normal,Normal/images/Normal-7899.png,Normal/masks/Normal-7899.png,True,G17271,train,ff428bf47aff8941a0093945b1d1d088,987f061a5f585c5c
Normal-79.png,Normal,Normal/images/Normal-79.png,Normal/masks/Normal-79.png,True,G17272,test,e2bb61fd8ee6138b2a496698fa217e25,d129c27ada697869
Normal-790.png,Normal,Normal/images/Normal-790.png,Normal/masks/Normal-790.png,True,G17273,train,e8a8ea7bea2d4057dca822309c27dcff,916faf0e094d69c5
Normal-7900.png,Normal,Normal/images/Normal-7900.png,Normal/masks/Normal-7900.png,True,G17274,train,1518cb52e0db97916a79d30a01072714,c1299e1d7eec68c2
Normal-7901.png,Normal,Normal/images/Normal-7901.png,Normal/masks/Normal-7901.png,True,G17275,train,048e2830c898d6114c9edb7c3d1a8c07,d4440a4f2c4d6f7d
Normal-7902.png,Normal,Normal/images/Normal-7902.png,Normal/masks/Normal-7902.png,True,G17276,train,ae8f8f95e7649caded7ca19d9d9920e5,dd6e471b490a0f0f
Normal-7903.png,Normal,Normal/images/Normal-7903.png,Normal/masks/Normal-7903.png,True,G17277,train,610ecb2b052ee4dcf7e380424e0dc997,d939c41a1f471b1e
Normal-7904.png,Normal,Normal/images/Normal-7904.png,Normal/masks/Normal-7904.png,True,G17278,val,e15ba3a551baa1ec1779f01f8fd00dce,dd3cd2434f0c4e0f
Normal-7905.png,Normal,Normal/images/Normal-7905.png,Normal/masks/Normal-7905.png,True,G17279,val,1cb374058eaf98290a2a543f2206ac1c,997bc50616c74f1a
Normal-7906.png,Normal,Normal/images/Normal-7906.png,Normal/masks/Normal-7906.png,True,G17280,train,579373c05b8b1cd13780f3e2132dbf5f,c83dc55293451e5f
Normal-7907.png,Normal,Normal/images/Normal-7907.png,Normal/masks/Normal-7907.png,True,G17281,test,2f60e51ffd798bc54d9af5ecbd39a06b,982c465ac57af8e5
Normal-7908.png,Normal,Normal/images/Normal-7908.png,Normal/masks/Normal-7908.png,True,G17282,test,c1970337ab5bde8b332e3dc5e71249a9,986dc71a13871f95
Normal-7909.png,Normal,Normal/images/Normal-7909.png,Normal/masks/Normal-7909.png,True,G17283,train,e1ef0253de8649085bfc95935c85a093,9b210e5e5b126b79
Normal-791.png,Normal,Normal/images/Normal-791.png,Normal/masks/Normal-791.png,True,G17284,train,6d11b0bcf6396c2b71eba04b75478c7c,c5fe8f19ec0e2886
Normal-7910.png,Normal,Normal/images/Normal-7910.png,Normal/masks/Normal-7910.png,True,G17285,train,1eaf0d321a1b92567060a49726b7cbdd,9d3f445f1b451c1c
Normal-7911.png,Normal,Normal/images/Normal-7911.png,Normal/masks/Normal-7911.png,True,G17286,train,d3870ab2091048ae845a2c5d2c3bf003,816886171f5e7a5b
Normal-7912.png,Normal,Normal/images/Normal-7912.png,Normal/masks/Normal-7912.png,True,G17287,train,26ac7a396e9da6af747ad901d84a6b32,d52cd6d74b082e4b
Normal-7913.png,Normal,Normal/images/Normal-7913.png,Normal/masks/Normal-7913.png,True,G17288,train,610c06ec03e84badb62b948d2ccf8903,d17b065e186f2d32
Normal-7914.png,Normal,Normal/images/Normal-7914.png,Normal/masks/Normal-7914.png,True,G17289,train,8fccd41d802f69e606b746bfdc70c78a,dd2a570a1d4d581f
Normal-7915.png,Normal,Normal/images/Normal-7915.png,Normal/masks/Normal-7915.png,True,G17290,val,ee06ee4302fa56a289aa08f8d9f72302,d92806061b5ecf7b
Normal-7916.png,Normal,Normal/images/Normal-7916.png,Normal/masks/Normal-7916.png,True,G17291,val,64407f3f1777627806f1dbaebaf083ea,81ebd616005a7d5f
Normal-7917.png,Normal,Normal/images/Normal-7917.png,Normal/masks/Normal-7917.png,True,G17292,test,a3bcf5ee843c6bada5f177dc6793a82c,dc2f524f0a0b1f2e
Normal-7918.png,Normal,Normal/images/Normal-7918.png,Normal/masks/Normal-7918.png,True,G17293,train,7115751c4d2a98ca2a931949819bd9a6,891f161a193bbbaa
Normal-7919.png,Normal,Normal/images/Normal-7919.png,Normal/masks/Normal-7919.png,True,G17294,train,e5a0e64143392ef350cd706197327e41,d059538b1d2dad3c
Normal-792.png,Normal,Normal/images/Normal-792.png,Normal/masks/Normal-792.png,True,G17295,train,cebb7fc552569e920bad33c234de0cd6,886dc65afa693c4c
Normal-7920.png,Normal,Normal/images/Normal-7920.png,Normal/masks/Normal-7920.png,True,G17296,train,83703f932cfaef63c4dc82121cddfe22,d969421b1d453bb9
Normal-7921.png,Normal,Normal/images/Normal-7921.png,Normal/masks/Normal-7921.png,True,G17297,train,4ef0486b81081bf9557bcc9a753de133,ddb4c21fc51c594c
Normal-7922.png,Normal,Normal/images/Normal-7922.png,Normal/masks/Normal-7922.png,True,G17298,train,25056c3a2ad2136cfd696fbe15a195f9,d128575b0b4b2b3b
Normal-7923.png,Normal,Normal/images/Normal-7923.png,Normal/masks/Normal-7923.png,True,G17299,train,7ff5eb724c327fefec34251531d07faa,dd6c871a1f455e48
Normal-7924.png,Normal,Normal/images/Normal-7924.png,Normal/masks/Normal-7924.png,True,G17300,val,c913c500d8fbbc47a90a9981df3d6364,9978061a1d4e7df8
Normal-7925.png,Normal,Normal/images/Normal-7925.png,Normal/masks/Normal-7925.png,True,G17301,test,05c962ace3432613773f31d4bc5a7d06,dd2c428b0a4cef3d
Normal-7926.png,Normal,Normal/images/Normal-7926.png,Normal/masks/Normal-7926.png,True,G17302,test,2aeba1f0233d8f03318af123cd43278b,cd3cd20f550e1b47
Normal-7927.png,Normal,Normal/images/Normal-7927.png,Normal/masks/Normal-7927.png,True,G17303,train,26c43f4aaf1c26fcc962fb6de8804d92,d07dc7911cc42e3d
Normal-7928.png,Normal,Normal/images/Normal-7928.png,Normal/masks/Normal-7928.png,True,G17304,train,21144f3af508bf46a4b313e951c2982b,997b84161a5d5b3a
Normal-7929.png,Normal,Normal/images/Normal-7929.png,Normal/masks/Normal-7929.png,True,G17305,train,8b3af4f241ddb52ebe5f94002f35ffe8,897856191b4f39b5
Normal-793.png,Normal,Normal/images/Normal-793.png,Normal/masks/Normal-793.png,True,G17306,train,ea9c75f2f76214a9c2f17f23c560474d,917e8f3a1d1d1c45
Normal-7930.png,Normal,Normal/images/Normal-7930.png,Normal/masks/Normal-7930.png,True,G17307,train,fe8a0e1daf97fa6025041aedc3a0331a,9c2dc31b450f4f4d
Normal-7931.png,Normal,Normal/images/Normal-7931.png,Normal/masks/Normal-7931.png,True,G17308,train,1435413ec2821899430fae05c386a6f6,997bd61a1a0daab8
Normal-7932.png,Normal,Normal/images/Normal-7932.png,Normal/masks/Normal-7932.png,True,G17309,train,552499e0a3c5cbee6385622a39a48693,dc2ec2434f4c6c3e
Normal-7933.png,Normal,Normal/images/Normal-7933.png,Normal/masks/Normal-7933.png,True,G17310,train,5a6b1eaaabeed0bbf4f92b00d3b6f07a,cd2dc24f1bc60b1b
Normal-7934.png,Normal,Normal/images/Normal-7934.png,Normal/masks/Normal-7934.png,True,G17311,train,2cae3639eb0693b90ecc1af3f5db7916,997c461b084e79ed
Normal-7935.png,Normal,Normal/images/Normal-7935.png,Normal/masks/Normal-7935.png,True,G17312,train,b283b783292479d92c0c67df445ab38e,9d3c531b144f19ce
Normal-7936.png,Normal,Normal/images/Normal-7936.png,Normal/masks/Normal-7936.png,True,G17313,train,8cfd291cd1ff8dcb0c458ed8afc52351,dd2d52571a464b1b
Normal-7937.png,Normal,Normal/images/Normal-7937.png,Normal/masks/Normal-7937.png,True,G17314,val,f9d4351a9e8de2723caf543e552e48ad,81e8ded6154d1a79
Normal-7938.png,Normal,Normal/images/Normal-7938.png,Normal/masks/Normal-7938.png,True,G17315,train,fa01563578aea0dbaf736e5e953486c6,d03dc29f5e424a5b
Normal-7939.png,Normal,Normal/images/Normal-7939.png,Normal/masks/Normal-7939.png,True,G17316,val,d88a17a0c5e031459c31a251be2a961c,c03cc3d34bc2cead
Normal-794.png,Normal,Normal/images/Normal-794.png,Normal/masks/Normal-794.png,True,G17317,val,ddec6131eb1fbd9f4467ab14a578f7e5,d13dc21ad87938d9
Normal-7940.png,Normal,Normal/images/Normal-7940.png,Normal/masks/Normal-7940.png,True,G17318,test,1a9b137247b795b21830ff8391690c14,8169c61e1d5e5e3a
Normal-7941.png,Normal,Normal/images/Normal-7941.png,Normal/masks/Normal-7941.png,True,G17319,test,92d90b3b73f4a61c9f50827b07388243,806847c61b5f4d5f
Normal-7942.png,Normal,Normal/images/Normal-7942.png,Normal/masks/Normal-7942.png,True,G17320,test,e3974ff4f0d97ce2588bacfa380b44d3,987d861e18cd3ab9
Normal-7943.png,Normal,Normal/images/Normal-7943.png,Normal/masks/Normal-7943.png,True,G17321,train,b4e39400445c5ffa84bb2bf7404497b4,c93c461b39ece9b0
Normal-7944.png,Normal,Normal/images/Normal-7944.png,Normal/masks/Normal-7944.png,True,G17322,train,aaee12d84c2be7de5e8e363dd10102c9,817d065a1dec7bb0
Normal-7945.png,Normal,Normal/images/Normal-7945.png,Normal/masks/Normal-7945.png,True,G17323,train,d6d711c1e2ccce89e6c5de63b5f33edd,cc2d820b0eee6f3c
Normal-7946.png,Normal,Normal/images/Normal-7946.png,Normal/masks/Normal-7946.png,True,G17324,train,1410def72b290725cca5c597d5560a72,89ff061a971a194f
Normal-7947.png,Normal,Normal/images/Normal-7947.png,Normal/masks/Normal-7947.png,True,G17325,train,f1420c4c0705bdd7ed235562a0544324,952c86831b5f0f1f
Normal-7948.png,Normal,Normal/images/Normal-7948.png,Normal/masks/Normal-7948.png,True,G17326,train,96b948043fcf9f063a304c712f4b2bbd,d43c834b2d4a3d7c
Normal-7949.png,Normal,Normal/images/Normal-7949.png,Normal/masks/Normal-7949.png,True,G17327,train,a3253ac6b8e11bbc48e2f2db5ede41ce,dd3dc60b1d481a5d
Normal-795.png,Normal,Normal/images/Normal-795.png,Normal/masks/Normal-795.png,True,G17328,train,a7cd7026ab836357752ce36bd7b42510,d0eaab8bd5c5b0a2
Normal-7950.png,Normal,Normal/images/Normal-7950.png,Normal/masks/Normal-7950.png,True,G17329,val,c8b2cf18ad332c7c00f3b14c52991e8b,d8e905961fc83eca
Normal-7951.png,Normal,Normal/images/Normal-7951.png,Normal/masks/Normal-7951.png,True,G17330,test,a7b53f6d6d41147e51be813b311eb39e,883dc30caebdb4a6
Normal-7952.png,Normal,Normal/images/Normal-7952.png,Normal/masks/Normal-7952.png,True,G17331,val,8ff7e6e60c43aa06f5604148dd75a80f,dc3cd3431f4f4149
Normal-7953.png,Normal,Normal/images/Normal-7953.png,Normal/masks/Normal-7953.png,True,G17332,train,3ac772a6248c1eb0413d8e9ec8bfc53d,c13986471d5e6979
Normal-7954.png,Normal,Normal/images/Normal-7954.png,Normal/masks/Normal-7954.png,True,G17333,val,64e18af2c3bdcb36bd56782c1692fda2,dd2a434f484f0a7d
Normal-7955.png,Normal,Normal/images/Normal-7955.png,Normal/masks/Normal-7955.png,True,G17334,train,485b5115c660443b3856a10f115372d2,d42d42475f5b4d0d
Normal-7956.png,Normal,Normal/images/Normal-7956.png,Normal/masks/Normal-7956.png,True,G17335,train,36687e401e5837f936a8d091ccfe41e3,d42f46431b0bcd7a
Normal-7957.png,Normal,Normal/images/Normal-7957.png,Normal/masks/Normal-7957.png,True,G17336,train,ef7af814f806f99722fb1dfa22a8ecad,897907065e17ce5e
Normal-7958.png,Normal,Normal/images/Normal-7958.png,Normal/masks/Normal-7958.png,True,G17337,train,95a13b64b6794224df21231696715f01,dc2e525b420b0f6f
Normal-7959.png,Normal,Normal/images/Normal-7959.png,Normal/masks/Normal-7959.png,True,G17338,test,a0a0abe2290b53c10c480ed0edc4b104,9d7cc61b19451e1d
Normal-796.png,Normal,Normal/images/Normal-796.png,Normal/masks/Normal-796.png,True,G17339,test,61593a6db3c358bab43ccbcad5639cff,8c2e071a5d4d3e6d
Normal-7960.png,Normal,Normal/images/Normal-7960.png,Normal/masks/Normal-7960.png,True,G17340,val,8f1292ec24ac3809f8432bd13478343c,dd6ad6181c1b2ce5
Normal-7961.png,Normal,Normal/images/Normal-7961.png,Normal/masks/Normal-7961.png,True,G17341,train,bf64a43abc0bacf3f3780168fa1ce94f,817b4686de1af84d
Normal-7962.png,Normal,Normal/images/Normal-7962.png,Normal/masks/Normal-7962.png,True,G17342,train,7b913abe0b35e8ff059591b3b20437bf,dc3cc21bc00f9b4f
Normal-7963.png,Normal,Normal/images/Normal-7963.png,Normal/masks/Normal-7963.png,True,G17343,train,fc743f365a195e3f924b2ae5992cb5c0,d93d871b855e184e
Normal-7964.png,Normal,Normal/images/Normal-7964.png,Normal/masks/Normal-7964.png,True,G17344,train,67efc9f2bab37799d83e820c1663ebc5,d83dc6549b02af1b
Normal-7965.png,Normal,Normal/images/Normal-7965.png,Normal/masks/Normal-7965.png,True,G17345,train,2f28e78a1c07463257c160e14927a7c9,9d7c464b196da43a
Normal-7966.png,Normal,Normal/images/Normal-7966.png,Normal/masks/Normal-7966.png,True,G17346,train,32cd1dd7099d8c492e4b0cd40c537019,993d421a8d6abbe8
Normal-7967.png,Normal,Normal/images/Normal-7967.png,Normal/masks/Normal-7967.png,True,G17347,train,c76b8120646e23a51f8e46f033838c86,dc2cd24b5f420b4f
Normal-7968.png,Normal,Normal/images/Normal-7968.png,Normal/masks/Normal-7968.png,True,G17348,train,dc4f8188ec107dd5b472497a44c500e5,94bfc7190e4e2d0e
Normal-7969.png,Normal,Normal/images/Normal-7969.png,Normal/masks/Normal-7969.png,True,G17349,train,ec70b4881186a4436a123d57ea621dc2,992e44da59edb382
Normal-797.png,Normal,Normal/images/Normal-797.png,Normal/masks/Normal-797.png,True,G17350,train,7ab3cfd7569bd0fa2f283c460ddd05ce,956a8758cd3d82c7
Normal-7970.png,Normal,Normal/images/Normal-7970.png,Normal/masks/Normal-7970.png,True,G17351,train,02a8270e149749ee9cfbd352ab760be5,886b47065e5d4e5e
Normal-7971.png,Normal,Normal/images/Normal-7971.png,Normal/masks/Normal-7971.png,True,G17352,train,f0f6460f1bfcde3830c8709661f3c58a,917bd618091b39bb
Normal-7972.png,Normal,Normal/images/Normal-7972.png,Normal/masks/Normal-7972.png,True,G17353,train,18bdb645b47c5ee3b705b70c94fcd6ea,d979574619131e3a
Normal-7973.png,Normal,Normal/images/Normal-7973.png,Normal/masks/Normal-7973.png,True,G17354,train,853c87f9a99346b5bc0a1b85141dd368,d948831f584dbf15
Normal-7974.png,Normal,Normal/images/Normal-7974.png,Normal/masks/Normal-7974.png,True,G17355,train,52056355fc638ea85ef2f5839c3f2f7d,9b788d1c185b7a5a
Normal-7975.png,Normal,Normal/images/Normal-7975.png,Normal/masks/Normal-7975.png,True,G17356,val,eeb2b85f21a4fa88b4aa199d89849f2c,9c7d435b494c2c7c
Normal-7976.png,Normal,Normal/images/Normal-7976.png,Normal/masks/Normal-7976.png,True,G17357,val,99d119b09730b68a2393076fab6792e4,cd2d820f9d0e1bec
Normal-7977.png,Normal,Normal/images/Normal-7977.png,Normal/masks/Normal-7977.png,True,G17358,test,7103856f20881660d6a0b53682c67c2f,d62dd5d14b436964
Normal-7978.png,Normal,Normal/images/Normal-7978.png,Normal/masks/Normal-7978.png,True,G17359,test,9feaaabb423a52f4b058bf52f86e1ecd,dd2a470e0d0d4d6f
Normal-7979.png,Normal,Normal/images/Normal-7979.png,Normal/masks/Normal-7979.png,True,G17360,test,9a696c9e2c30a10a5985622fd9c5203d,95681647195eed78
Normal-798.png,Normal,Normal/images/Normal-798.png,Normal/masks/Normal-798.png,True,G17361,train,a28462fc9c07b4e64d5a7ae9584814fc,dcacc21bcb4f1cc2
Normal-7980.png,Normal,Normal/images/Normal-7980.png,Normal/masks/Normal-7980.png,True,G17362,train,53b49d56f1aa616279e3cb2eaeee5a1a,c62fc14b0ceed10f
Normal-7981.png,Normal,Normal/images/Normal-7981.png,Normal/masks/Normal-7981.png,True,G17363,train,f43098f48d40230a4e1ed405aa6fe865,8929575f1c0e3d39
Normal-7982.png,Normal,Normal/images/Normal-7982.png,Normal/masks/Normal-7982.png,True,G17364,train,a03c4f4cd9faacf4d2fb54f393538957,957ec603598f2ab8
Normal-7983.png,Normal,Normal/images/Normal-7983.png,Normal/masks/Normal-7983.png,True,G17365,train,df785449dbd3a64418a65766e753e027,f97984d61a5d445a
Normal-7984.png,Normal,Normal/images/Normal-7984.png,Normal/masks/Normal-7984.png,True,G17366,train,29bb8885bf0e35d4242e24b4039a0575,d47f524b4b4d0a0f
Normal-7985.png,Normal,Normal/images/Normal-7985.png,Normal/masks/Normal-7985.png,True,G17367,train,c90c404d40a5de97522ada1ccef9ee65,987d861a194d7ce9
Normal-7986.png,Normal,Normal/images/Normal-7986.png,Normal/masks/Normal-7986.png,True,G17368,train,bf1a5f6f7a0e0309db2aad2d57c6eaf7,d43c0f0b0f6e3c69
Normal-7987.png,Normal,Normal/images/Normal-7987.png,Normal/masks/Normal-7987.png,True,G17369,train,574e0a6fd917966db58cc17415c065f5,c93bc70e1a854b3b
Normal-7988.png,Normal,Normal/images/Normal-7988.png,Normal/masks/Normal-7988.png,True,G17370,train,eecd5e4d98b75407b315f75b884e6de8,81695616185ecf7b
Normal-7989.png,Normal,Normal/images/Normal-7989.png,Normal/masks/Normal-7989.png,True,G17371,train,a7f6bce44fec12678fb3e603e6a18b0f,f979871e184f185a
Normal-799.png,Normal,Normal/images/Normal-799.png,Normal/masks/Normal-799.png,True,G17372,train,8c1e45bc7383d00915a1979e7f88f612,917dc71a9d4918cd
Normal-7990.png,Normal,Normal/images/Normal-7990.png,Normal/masks/Normal-7990.png,True,G17373,train,acb4254be108f9e654e4ec9257f79418,887906561d5d7bb8
Normal-7991.png,Normal,Normal/images/Normal-7991.png,Normal/masks/Normal-7991.png,True,G17374,train,19075178f411c069a69b9ef32faeba30,dd2cd2031fc24b1f
Normal-7992.png,Normal,Normal/images/Normal-7992.png,Normal/masks/Normal-7992.png,True,G17375,train,ff4bef088ab2b2f0d27132668bd3f7df,9169169e79e9b815
Normal-7993.png,Normal,Normal/images/Normal-7993.png,Normal/masks/Normal-7993.png,True,G17376,train,9322712b3554f8b6f7d612dc025395e0,993fc71017561e4e
Normal-7994.png,Normal,Normal/images/Normal-7994.png,Normal/masks/Normal-7994.png,True,G17377,val,25c217bca672d1f1b913a26b12a7d092,9978c6171f561e1a
Normal-7995.png,Normal,Normal/images/Normal-7995.png,Normal/masks/Normal-7995.png,True,G17378,val,80d6e7b634e9e608ba8e00ae32063930,dcbce21bc14596e2
Normal-7996.png,Normal,Normal/images/Normal-7996.png,Normal/masks/Normal-7996.png,True,G17379,val,a91ea9b615f0175e3620b56286d01d6a,dd38c2db1d401b1f
Normal-7997.png,Normal,Normal/images/Normal-7997.png,Normal/masks/Normal-7997.png,True,G17380,test,81df49f1ca313859663b880c1fae128a,997d87129f871b88
Normal-7998.png,Normal,Normal/images/Normal-7998.png,Normal/masks/Normal-7998.png,True,G17381,test,eeab07b4ee1b26bc16b431c1d308ac31,dc6a531b4d0e0d2f
Normal-7999.png,Normal,Normal/images/Normal-7999.png,Normal/masks/Normal-7999.png,True,G17382,test,322ccc48b9955166ebc7c9a25ae25d3a,d42e42530f4f4f4e
Normal-8.png,Normal,Normal/images/Normal-8.png,Normal/masks/Normal-8.png,True,G17383,train,b0c39901dd2c6a4985ac723a7d9a0fae,dd7c46535b4b0919
Normal-80.png,Normal,Normal/images/Normal-80.png,Normal/masks/Normal-80.png,True,G17384,val,7b605554e8cdc566865d112585b573c7,d05ec71bc9e568e0
Normal-800.png,Normal,Normal/images/Normal-800.png,Normal/masks/Normal-800.png,True,G17385,train,3709d3e9490c8c93eebc9fa8f84fd030,913dc31ab96939e1
Normal-8000.png,Normal,Normal/images/Normal-8000.png,Normal/masks/Normal-8000.png,True,G17386,train,5cbe96f12ad6af3d7407856602fc6c45,c03dd30b4a4b2b6f
Normal-8001.png,Normal,Normal/images/Normal-8001.png,Normal/masks/Normal-8001.png,True,G17387,train,03edc1d2722a3ef9d33d7d1d9decec11,847d020b1b4f4f5f
Normal-8002.png,Normal,Normal/images/Normal-8002.png,Normal/masks/Normal-8002.png,True,G17388,train,16ff10c6188bd35c98813a6b6f7af4ab,9c07031f1e1e5e5e
Normal-8003.png,Normal,Normal/images/Normal-8003.png,Normal/masks/Normal-8003.png,True,G17389,train,d44d9cb757670ed9b16c16efa691f78d,89391e1d59efac82
Normal-8004.png,Normal,Normal/images/Normal-8004.png,Normal/masks/Normal-8004.png,True,G17390,train,b37aaab055c0a237bbdafc8a05e79b87,d97b0606171e1f1e
Normal-8005.png,Normal,Normal/images/Normal-8005.png,Normal/masks/Normal-8005.png,True,G17391,train,8c609f057e6f39f42e2962ebf494c7ee,c969061e16c75f9c
Normal-8006.png,Normal,Normal/images/Normal-8006.png,Normal/masks/Normal-8006.png,True,G17392,train,4b909f0483e7e1971537d8d1fba5da45,892cc613fb4ff811
Normal-8007.png,Normal,Normal/images/Normal-8007.png,Normal/masks/Normal-8007.png,True,G17393,train,c1d78ea3e658360c09751aa04b8c685c,d42f42575b144b1f
Normal-8008.png,Normal,Normal/images/Normal-8008.png,Normal/masks/Normal-8008.png,True,G17394,train,2bd7e897c3674ef5c3ae462bdc17216e,9979861e1b575a19
Normal-8009.png,Normal,Normal/images/Normal-8009.png,Normal/masks/Normal-8009.png,True,G17395,train,1b47d1bf2e9718afccf4c4a51d7c08aa,dc3d431d1a0f3879
Normal-801.png,Normal,Normal/images/Normal-801.png,Normal/masks/Normal-801.png,True,G17396,train,d80c6ef7877555315911345dd88c273e,d57e872f0d2d0cc4
Normal-8010.png,Normal,Normal/images/Normal-8010.png,Normal/masks/Normal-8010.png,True,G17397,train,6674d78fb729b10f67c8dbe1123a378b,d92c861f0d4f1bcc
Normal-8011.png,Normal,Normal/images/Normal-8011.png,Normal/masks/Normal-8011.png,True,G17398,train,eec8ee4fe8e1a491aac8eb08da135026,8068471e5f4d7ad9
Normal-8012.png,Normal,Normal/images/Normal-8012.png,Normal/masks/Normal-8012.png,True,G17399,val,9b17d1286609bb473eb5845f05b74d95,d139460a3b25e3bb
Normal-8013.png,Normal,Normal/images/Normal-8013.png,Normal/masks/Normal-8013.png,True,G17400,val,8fcb14dabd1a162d98713bad31c81e0c,dd7d02124f0a3a7d
Normal-8014.png,Normal,Normal/images/Normal-8014.png,Normal/masks/Normal-8014.png,True,G17401,test,c817df36182bdb0f8b944bad0c5f7784,dc0b430deabca9b1
Normal-8015.png,Normal,Normal/images/Normal-8015.png,Normal/masks/Normal-8015.png,True,G17402,test,d014668c7ce1b328e966e26063b9156b,d93d06185d4f78d1
Normal-8016.png,Normal,Normal/images/Normal-8016.png,Normal/masks/Normal-8016.png,True,G17403,test,f761754988f73a0119bb1ea6166c1e17,c87dc21f5c495d4a
Normal-8017.png,Normal,Normal/images/Normal-8017.png,Normal/masks/Normal-8017.png,True,G17404,train,598eb64e1240bf709964a2e6171ef14d,9d3cc65b1b4c6c38
Normal-8018.png,Normal,Normal/images/Normal-8018.png,Normal/masks/Normal-8018.png,True,G17405,train,9c06cb7063c0006f2a360fbb6ae59eb5,c82e865f1d78693c
Normal-8019.png,Normal,Normal/images/Normal-8019.png,Normal/masks/Normal-8019.png,True,G17406,train,c67867248417124ef099758a314e2c9c,8c0a131e5c5a5f7b
Normal-802.png,Normal,Normal/images/Normal-802.png,Normal/masks/Normal-802.png,True,G17407,val,a8b8f4a75681d3361255164696394d09,997dc75a8e4b184a
Normal-8020.png,Normal,Normal/images/Normal-8020.png,Normal/masks/Normal-8020.png,True,G17408,train,9392ebcb8c6f03e8414cd2d480929b3a,983f45c61e59a995
Normal-8021.png,Normal,Normal/images/Normal-8021.png,Normal/masks/Normal-8021.png,True,G17409,train,f7436b38e4d2ac4976ef427ff3a06ebe,9968860b1bd67ab9
Normal-8022.png,Normal,Normal/images/Normal-8022.png,Normal/masks/Normal-8022.png,True,G17410,train,9896495bab7bae3b7bad73481722e051,d42dc3435f0aad39
Normal-8023.png,Normal,Normal/images/Normal-8023.png,Normal/masks/Normal-8023.png,True,G17411,train,c3a5d21e5304c8dde32a16c1f464aa9a,817b061a1b79e8f9
Normal-8024.png,Normal,Normal/images/Normal-8024.png,Normal/masks/Normal-8024.png,True,G17412,train,cc4fed3b5c70431de7a69e09552c0f50,91ed861ac70e1b1f
Normal-8025.png,Normal,Normal/images/Normal-8025.png,Normal/masks/Normal-8025.png,True,G17413,train,a3397e18da5e47f184ff5a1cf06e9085,997b851e1b585a69
Normal-8026.png,Normal,Normal/images/Normal-8026.png,Normal/masks/Normal-8026.png,True,G17414,train,75909f3d7d0d265238774997c2219eef,997807061adede5a
Normal-8027.png,Normal,Normal/images/Normal-8027.png,Normal/masks/Normal-8027.png,True,G17415,train,25b92a0c2daacf8ac77c6d4817333688,d930063aed69b1e5
Normal-8028.png,Normal,Normal/images/Normal-8028.png,Normal/masks/Normal-8028.png,True,G17416,test,27047f3146d9eae13568b1d1a4553847,dc2fd24b0f440e3d
Normal-8029.png,Normal,Normal/images/Normal-8029.png,Normal/masks/Normal-8029.png,True,G17417,train,57dd91afb539a6bc1b69865b4abff0ee,c17c07521b4faa5d
Normal-803.png,Normal,Normal/images/Normal-803.png,Normal/masks/Normal-803.png,True,G17418,test,bf03c67444c51ae98548161e032df60f,902dc61abe693de4
Normal-8030.png,Normal,Normal/images/Normal-8030.png,Normal/masks/Normal-8030.png,True,G17419,train,25b11e445fb8542896ded58effb388ec,897b1716185d5b3a
Normal-8031.png,Normal,Normal/images/Normal-8031.png,Normal/masks/Normal-8031.png,True,G17420,test,2e2b3a6e0a594286c282f584d5659e89,991d061e1a5d5e4f
Normal-8032.png,Normal,Normal/images/Normal-8032.png,Normal/masks/Normal-8032.png,True,G17421,train,df281512d00e9c423a8a4e7abe7c71b7,dc3c430f8aae2db8
Normal-8033.png,Normal,Normal/images/Normal-8033.png,Normal/masks/Normal-8033.png,True,G17422,train,e2c1c75f7a2aad0fb8c60f4fd02b58aa,d96f460a4c0b3dc7
Normal-8034.png,Normal,Normal/images/Normal-8034.png,Normal/masks/Normal-8034.png,True,G17423,val,a833abf4fa5c32ec1bce369e7d074803,dd7d06821b0ac74f
Normal-8035.png,Normal,Normal/images/Normal-8035.png,Normal/masks/Normal-8035.png,True,G17424,train,1bff4d794e9bcec8163a1c9b225e8fa5,d93db687091b424f
Normal-8036.png,Normal,Normal/images/Normal-8036.png,Normal/masks/Normal-8036.png,True,G17425,val,62f7c910b52463f37bdb8b9a610d77fc,81e86e875ad94d6c
Normal-8037.png,Normal,Normal/images/Normal-8037.png,Normal/masks/Normal-8037.png,True,G17426,train,08e1dd06c5f6995648fee940415ae86f,887996d65e1c5d49
Normal-8038.png,Normal,Normal/images/Normal-8038.png,Normal/masks/Normal-8038.png,True,G17427,train,7cc59786c8b2af3944e93386950059f3,81785e171a1d5e5e
Normal-8039.png,Normal,Normal/images/Normal-8039.png,Normal/masks/Normal-8039.png,True,G17428,train,0ea03332b7172e26a9c32ba39d1716b2,807c571a5d693c5d
Normal-804.png,Normal,Normal/images/Normal-804.png,Normal/masks/Normal-804.png,True,G17429,train,d32b538214df04593143d9c00ee14053,912dce3ad959184f
Normal-8040.png,Normal,Normal/images/Normal-8040.png,Normal/masks/Normal-8040.png,True,G17430,train,3fd5d66c2bfb3fb80d35bee3e45e17ca,cc7e4e1d1607b1a3
Normal-8041.png,Normal,Normal/images/Normal-8041.png,Normal/masks/Normal-8041.png,True,G17431,train,cc29cf7ada13502a44c76dcf93d19399,d969071e1a4d5b59
Normal-8042.png,Normal,Normal/images/Normal-8042.png,Normal/masks/Normal-8042.png,True,G17432,train,7b55d99c2e4941336472ecc7df951893,dd2846471b1f0f4d
Normal-8043.png,Normal,Normal/images/Normal-8043.png,Normal/masks/Normal-8043.png,True,G17433,train,c4131df2b3de3412844502534676f6c8,987d05561e5d491f
Normal-8044.png,Normal,Normal/images/Normal-8044.png,Normal/masks/Normal-8044.png,True,G17434,train,14d6b91f499f2440372e45562fa76dbc,d13dc03b78587978
Normal-8045.png,Normal,Normal/images/Normal-8045.png,Normal/masks/Normal-8045.png,True,G17435,train,c0cadd25a58461885bae287a46712e3d,996f061a4d194f5e
Normal-8046.png,Normal,Normal/images/Normal-8046.png,Normal/masks/Normal-8046.png,True,G17436,train,728855d6acbd50033c8e24b9716b4786,8178c756185a4f5f
Normal-8047.png,Normal,Normal/images/Normal-8047.png,Normal/masks/Normal-8047.png,True,G17437,train,6ea4937940aa2a4e8266e27da1e52c1f,dd28d64b1f414b1d
Normal-8048.png,Normal,Normal/images/Normal-8048.png,Normal/masks/Normal-8048.png,True,G17438,val,fb1bdeb3f4f32b24a512718cfc467a4e,d92fc41f1ac42bb8
Normal-8049.png,Normal,Normal/images/Normal-8049.png,Normal/masks/Normal-8049.png,True,G17439,val,3bc2568f8aa59062ccb956157867fca2,8178465a5acdf8b5
Normal-805.png,Normal,Normal/images/Normal-805.png,Normal/masks/Normal-805.png,True,G17440,val,48ee98adef1c363933271a1fd5ef0e4e,8121c61eec6d79e5
Normal-8050.png,Normal,Normal/images/Normal-8050.png,Normal/masks/Normal-8050.png,True,G17441,test,0b53ea81d658a36e7cdc4e67cfba6aea,990d061b5c7e7971
Normal-8051.png,Normal,Normal/images/Normal-8051.png,Normal/masks/Normal-8051.png,True,G17442,test,baae4c0758ed4688b15332d9a1e6eb24,cc5d420b0e2f2fbc
Normal-8052.png,Normal,Normal/images/Normal-8052.png,Normal/masks/Normal-8052.png,True,G17443,test,1a3527755d72a7ba7ccae3fb8159c86c,817a1618195f5b5f
Normal-8053.png,Normal,Normal/images/Normal-8053.png,Normal/masks/Normal-8053.png,True,G17444,train,aef879b278d0ced33e43f5ab1fcaf443,d13c2acf34f0634b
Normal-8054.png,Normal,Normal/images/Normal-8054.png,Normal/masks/Normal-8054.png,True,G17445,train,495fe5b850cfd0898756731e56ac3891,8169571e1e4f3e91
Normal-8055.png,Normal,Normal/images/Normal-8055.png,Normal/masks/Normal-8055.png,True,G17446,train,cdd109921e8c3dae7f2516636d18d5ca,d929c5561f455a1d
Normal-8056.png,Normal,Normal/images/Normal-8056.png,Normal/masks/Normal-8056.png,True,G17447,train,1d92a1a2228e8f1ba4dfc0a7ec0b6448,958d6e9a61639636
Normal-8057.png,Normal,Normal/images/Normal-8057.png,Normal/masks/Normal-8057.png,True,G17448,train,fca0949310657b64b5489ab951caf51b,d16a065f0e1b5f2a
Normal-8058.png,Normal,Normal/images/Normal-8058.png,Normal/masks/Normal-8058.png,True,G17449,train,4e1fdebdd08cac80b73e4cfb21029cda,997dc74a194d1d19
Normal-8059.png,Normal,Normal/images/Normal-8059.png,Normal/masks/Normal-8059.png,True,G17450,train,6c9bd0fc1d32b998778eb86e5bb5d75d,897b861a194b7ba9
Normal-806.png,Normal,Normal/images/Normal-806.png,Normal/masks/Normal-806.png,True,G17451,train,428080c4583581bc2e8b9640c9f582c2,9c75871aab2f3c24
Normal-8060.png,Normal,Normal/images/Normal-8060.png,Normal/masks/Normal-8060.png,True,G17452,train,2852a92024eeb67f82b0bc5ad5692679,cc2d42570fcb1b4d
Normal-8061.png,Normal,Normal/images/Normal-8061.png,Normal/masks/Normal-8061.png,True,G17453,train,a8e3b27f176d89f3594c1251521d7d4b,992f44461f4ded68
Normal-8062.png,Normal,Normal/images/Normal-8062.png,Normal/masks/Normal-8062.png,True,G17454,train,241c0ea693ecef13438d6771efe58c5c,917e8716194d4f0f
Normal-8063.png,Normal,Normal/images/Normal-8063.png,Normal/masks/Normal-8063.png,True,G17455,val,fe23534825cb214908a8ce6327418901,817ad61a3b197a1b
Normal-8064.png,Normal,Normal/images/Normal-8064.png,Normal/masks/Normal-8064.png,True,G17456,test,e3219259433f9042847da80585461bd1,817b56181b3b1b3b
Normal-8065.png,Normal,Normal/images/Normal-8065.png,Normal/masks/Normal-8065.png,True,G17457,train,12666ffd34d2a25b61b119367f9c3344,887d47c6521a5f4d
Normal-8066.png,Normal,Normal/images/Normal-8066.png,Normal/masks/Normal-8066.png,True,G17458,train,189d9c069499e2886a91496c77c0d8ff,883fd71a1d483a5d
Normal-8067.png,Normal,Normal/images/Normal-8067.png,Normal/masks/Normal-8067.png,True,G17459,train,e86dc0cc229aede85d60228ca3d29eea,dd29c71a1e464f1c
Normal-8068.png,Normal,Normal/images/Normal-8068.png,Normal/masks/Normal-8068.png,True,G17460,val,ff5f424c5f243b3298bf393bfaaa8b41,95294e5a79656363
Normal-8069.png,Normal,Normal/images/Normal-8069.png,Normal/masks/Normal-8069.png,True,G17461,val,8a26355ea29c0ab36eeb18ac497f43bc,897f8618195d195f
Normal-807.png,Normal,Normal/images/Normal-807.png,Normal/masks/Normal-807.png,True,G17462,test,324ed55e49b1d5fec7ab2abb8c14e418,91ecc61ac94d3cd6
Normal-8070.png,Normal,Normal/images/Normal-8070.png,Normal/masks/Normal-8070.png,True,G17463,test,4942a144e5a7ba8ef8eb612e49a7d4cf,91699e191f5e38e8
Normal-8071.png,Normal,Normal/images/Normal-8071.png,Normal/masks/Normal-8071.png,True,G17464,train,9aed9a12bbd9714f9cd2eb65fe3ab9a9,d93fc61c541e184f
Normal-8072.png,Normal,Normal/images/Normal-8072.png,Normal/masks/Normal-8072.png,True,G17465,train,8390d4e568b1ab5e1195a501dfc0f3db,b83f871e18455f1c
Normal-8073.png,Normal,Normal/images/Normal-8073.png,Normal/masks/Normal-8073.png,True,G17466,train,d54d6e4afd7879fe921992e01c4980d2,8c6ec2124f44ffe1
Normal-8074.png,Normal,Normal/images/Normal-8074.png,Normal/masks/Normal-8074.png,True,G17467,train,118f6c4cb32a6ab5bedeb6ca96249a85,9128069a5f9a79cf
Normal-8075.png,Normal,Normal/images/Normal-8075.png,Normal/masks/Normal-8075.png,True,G17468,train,316fd6027b5d5cea2243184f39f7726c,896857171a5d6979
Normal-8076.png,Normal,Normal/images/Normal-8076.png,Normal/masks/Normal-8076.png,True,G17469,train,7d632636b933bbd8ebb91be1796a9525,dd2ac30f1d443e5c
Normal-8077.png,Normal,Normal/images/Normal-8077.png,Normal/masks/Normal-8077.png,True,G17470,train,c70bd4f6016ba38ede981035f2b1cad7,d93d82171bce1f18
Normal-8078.png,Normal,Normal/images/Normal-8078.png,Normal/masks/Normal-8078.png,True,G17471,train,b9b21b436788f13dacfd166f213f30cc,817857961a1f7a5a
Normal-8079.png,Normal,Normal/images/Normal-8079.png,Normal/masks/Normal-8079.png,True,G17472,train,5a74ee239d33ea4d192f00a0eef5ac23,8079561e1f0f1f1b
Normal-808.png,Normal,Normal/images/Normal-808.png,Normal/masks/Normal-808.png,True,G17473,train,f3a85b118912d044d1bc8ed292a51a57,916f8e1a0d4d5dc5
Normal-8080.png,Normal,Normal/images/Normal-8080.png,Normal/masks/Normal-8080.png,True,G17474,train,ed1ca4e2c51ceea3c59c438511822eda,986d461b4d5e584f
Normal-8081.png,Normal,Normal/images/Normal-8081.png,Normal/masks/Normal-8081.png,True,G17475,train,6077e951552800796dd23ddfb4709e5f,c83dd2471f124f1d
Normal-8082.png,Normal,Normal/images/Normal-8082.png,Normal/masks/Normal-8082.png,True,G17476,train,1c268c33af7f98cbf4b5400682ac9a85,dc2d421d59623cf5
Normal-8083.png,Normal,Normal/images/Normal-8083.png,Normal/masks/Normal-8083.png,True,G17477,train,37daa01a162ba29b7c267f036002276f,c1add65fd44b1843
Normal-8084.png,Normal,Normal/images/Normal-8084.png,Normal/masks/Normal-8084.png,True,G17478,test,11491a98a3331d4cb4e61e9ce355aa8b,997b04861f596979
Normal-8085.png,Normal,Normal/images/Normal-8085.png,Normal/masks/Normal-8085.png,True,G17479,val,6f3953b36097f9b952d13158ad10a9ac,99fdc71a841e384d
Normal-8086.png,Normal,Normal/images/Normal-8086.png,Normal/masks/Normal-8086.png,True,G17480,val,a882ee8e3911270db424615327d2cb04,95784a9b7924a6e6
Normal-8087.png,Normal,Normal/images/Normal-8087.png,Normal/masks/Normal-8087.png,True,G17481,test,a54297370fa5f075a691225021794a17,992c841658ed5efa
Normal-8088.png,Normal,Normal/images/Normal-8088.png,Normal/masks/Normal-8088.png,True,G17482,test,5bdd8350cc8cbdd3f121e26165264f11,cd2ed21b440e5de6
Normal-8089.png,Normal,Normal/images/Normal-8089.png,Normal/masks/Normal-8089.png,True,G17483,train,8914d868fb1558941a9c4cf17aa44974,d13b06821f59ef54
Normal-809.png,Normal,Normal/images/Normal-809.png,Normal/masks/Normal-809.png,True,G17484,train,e8cedf0b551445ce13a51a407ebadfa6,916d871acb3d682d
Normal-8090.png,Normal,Normal/images/Normal-8090.png,Normal/masks/Normal-8090.png,True,G17485,train,9afda754b2aae58260f80b0ff69917f2,dc3d43091dec2cbc
Normal-8091.png,Normal,Normal/images/Normal-8091.png,Normal/masks/Normal-8091.png,True,G17486,train,b0e272f3f64cb52945a0f36da2d86ac8,807c071a7b657772
Normal-8092.png,Normal,Normal/images/Normal-8092.png,Normal/masks/Normal-8092.png,True,G17487,train,b4698ee9aa52e205a0ae1dceb5dcee20,997987061ac55f1b
Normal-8093.png,Normal,Normal/images/Normal-8093.png,Normal/masks/Normal-8093.png,True,G17488,train,00d9680ad369adce8c67432004b9564a,d929c61a1d433b5d
Normal-8094.png,Normal,Normal/images/Normal-8094.png,Normal/masks/Normal-8094.png,True,G17489,train,a01e690e82477b2846b637667a4fe619,997f125b0d180f3e
Normal-8095.png,Normal,Normal/images/Normal-8095.png,Normal/masks/Normal-8095.png,True,G17490,train,afa8b9068f41d94c9e1e3510e260a45a,dd29440616573e3f
Normal-8096.png,Normal,Normal/images/Normal-8096.png,Normal/masks/Normal-8096.png,True,G17491,val,73e1933cfbf121542dac782f678524c2,d43fc25f55404b1d
Normal-8097.png,Normal,Normal/images/Normal-8097.png,Normal/masks/Normal-8097.png,True,G17492,train,9c85501ac8a8d0d81dc9db32e2e5a3ea,b86d8796929b071e
Normal-8098.png,Normal,Normal/images/Normal-8098.png,Normal/masks/Normal-8098.png,True,G17493,train,3587087409a908f49005f03745111e34,817a56185a3d3b2f
Normal-8099.png,Normal,Normal/images/Normal-8099.png,Normal/masks/Normal-8099.png,True,G17494,train,ad69f7a2a05ab4ea6c4dd552cd68df40,d52a431b693e3c71
Normal-81.png,Normal,Normal/images/Normal-81.png,Normal/masks/Normal-81.png,True,G17495,train,d8efa4b8f8868e83c114ffcde6e5fecd,d01fc71a9cc136b6
Normal-810.png,Normal,Normal/images/Normal-810.png,Normal/masks/Normal-810.png,True,G17496,train,60b63f74575e19e605f7af944108bece,91a8c65ac87a7979
Normal-8100.png,Normal,Normal/images/Normal-8100.png,Normal/masks/Normal-8100.png,True,G17497,train,cf3f3b3361ca62c375931131204bdf3b,997f861a016e79c5
Normal-8101.png,Normal,Normal/images/Normal-8101.png,Normal/masks/Normal-8101.png,True,G17498,train,6e0714d73564d811eba4fbca94104469,895f161a195f1b8e
Normal-8102.png,Normal,Normal/images/Normal-8102.png,Normal/masks/Normal-8102.png,True,G17499,val,819e6d4dfc8fa8fa2c856468a2cd83c8,d17e06125b5d7a98
Normal-8103.png,Normal,Normal/images/Normal-8103.png,Normal/masks/Normal-8103.png,True,G17500,train,fb03ead140b68017b306f6c1a70c5d07,d92906d71e1d4e0f
Normal-8104.png,Normal,Normal/images/Normal-8104.png,Normal/masks/Normal-8104.png,True,G17501,val,95d1e6460c3efa7a8efd4961cd2f8090,9979045e586d78e9
Normal-8105.png,Normal,Normal/images/Normal-8105.png,Normal/masks/Normal-8105.png,True,G17502,test,7290268685dffe63b0452830b03c361c,cc3d420f0b2c3dbe
Normal-8106.png,Normal,Normal/images/Normal-8106.png,Normal/masks/Normal-8106.png,True,G17503,val,60d8aa02fdd9f890ff2e762a410b348c,dc3dd2531d860f19
Normal-8107.png,Normal,Normal/images/Normal-8107.png,Normal/masks/Normal-8107.png,True,G17504,test,5a9d413b4faaa300b89431cdd6e8cc0e,9c53c31a6d3a2e27
Normal-8108.png,Normal,Normal/images/Normal-8108.png,Normal/masks/Normal-8108.png,True,G17505,test,14b8fb933d730460961b481bc252f732,dc3d530b086b3db4
Normal-8109.png,Normal,Normal/images/Normal-8109.png,Normal/masks/Normal-8109.png,True,G17506,train,4efd7c8a30ea8847e6bdccbc1f4bc958,8979878a17815f1f
Normal-811.png,Normal,Normal/images/Normal-811.png,Normal/masks/Normal-811.png,True,G17507,train,c7990fc93337ec6e152a54cb8fcdc948,d5fdc73a8f0a3804
Normal-8110.png,Normal,Normal/images/Normal-8110.png,Normal/masks/Normal-8110.png,True,G17508,train,cfcea7aca01bcfda2017803602b35e6f,cc3cc30b1cce2dbc
Normal-8111.png,Normal,Normal/images/Normal-8111.png,Normal/masks/Normal-8111.png,True,G17509,train,17b79a1f624fc5504491bfd156a62c8c,d42c43430f2eaf2f
Normal-8112.png,Normal,Normal/images/Normal-8112.png,Normal/masks/Normal-8112.png,True,G17510,train,9d1e29f761e8bde4ef36619cc52b86b5,9179161f1e4a39b9
Normal-8113.png,Normal,Normal/images/Normal-8113.png,Normal/masks/Normal-8113.png,True,G17511,train,6221be26580b99059a93538de73f8b13,817f1e1969666ac6
Normal-8114.png,Normal,Normal/images/Normal-8114.png,Normal/masks/Normal-8114.png,True,G17512,train,6d4c4c3a3568518e1fcf71258b18cdca,9168161b4979e1df
Normal-8115.png,Normal,Normal/images/Normal-8115.png,Normal/masks/Normal-8115.png,True,G17513,val,6d3b510164cb9c909a50a47440bac103,883d471a18e8fdba
Normal-8116.png,Normal,Normal/images/Normal-8116.png,Normal/masks/Normal-8116.png,True,G17514,train,155f315325b91b023a3ec9b5905d5dff,fe6118587a61aba5
Normal-8117.png,Normal,Normal/images/Normal-8117.png,Normal/masks/Normal-8117.png,True,G17515,train,c69bd9cba2804b75ff0840eb0773c5ca,d12c061d5e6c7a6d
Normal-8118.png,Normal,Normal/images/Normal-8118.png,Normal/masks/Normal-8118.png,True,G17516,train,f007bb65986aab0129619aa491a85d1b,c17b161e5c293f26
Normal-8119.png,Normal,Normal/images/Normal-8119.png,Normal/masks/Normal-8119.png,True,G17517,train,2e1654264fba0d2842ca3d6501d504a3,dd2b52571f082b39
Normal-812.png,Normal,Normal/images/Normal-812.png,Normal/masks/Normal-812.png,True,G17518,val,8bdc121b7b8ce5f956f6378538f2cb48,d839c61a9a5939d9
Normal-8120.png,Normal,Normal/images/Normal-8120.png,Normal/masks/Normal-8120.png,True,G17519,train,d8715620d3c707b8b975b61709f73c0a,dc2dc34b0b0b2e6e
Normal-8121.png,Normal,Normal/images/Normal-8121.png,Normal/masks/Normal-8121.png,True,G17520,train,530fe396b0da4ad195919c8d1f8f871c,d979851e1e511b1d
Normal-8122.png,Normal,Normal/images/Normal-8122.png,Normal/masks/Normal-8122.png,True,G17521,val,b3ba42064990d744392dfb80c9b35774,887f461b1c6b3da8
Normal-8123.png,Normal,Normal/images/Normal-8123.png,Normal/masks/Normal-8123.png,True,G17522,test,3e6f1505fbb8a964b562796018c0c45e,dd2fc20b1d420f3d
Normal-8124.png,Normal,Normal/images/Normal-8124.png,Normal/masks/Normal-8124.png,True,G17523,test,4317f22b12d868bbe640322d84670442,917b841e595b5a3a
Normal-8125.png,Normal,Normal/images/Normal-8125.png,Normal/masks/Normal-8125.png,True,G17524,test,fc96527d723b00ad135b09911244154e,9979c4861b1d8e1f
Normal-8126.png,Normal,Normal/images/Normal-8126.png,Normal/masks/Normal-8126.png,True,G17525,train,8ba4ea7c16b870835a84a22ae07e00c8,d43dc2435b4b8e1b
Normal-8127.png,Normal,Normal/images/Normal-8127.png,Normal/masks/Normal-8127.png,True,G17526,train,152615c325f82742d6ea4d3a27e07d20,d829d5421f4d5a1f
Normal-8128.png,Normal,Normal/images/Normal-8128.png,Normal/masks/Normal-8128.png,True,G17527,train,dd44c2ea5faa73a56a5f4a7dbd0b9101,997d875e0c451a6d
Normal-8129.png,Normal,Normal/images/Normal-8129.png,Normal/masks/Normal-8129.png,True,G17528,train,fbda8c76016c3a43da0d47e8c6f5f77d,dd7cc21b154a1dac
Normal-813.png,Normal,Normal/images/Normal-813.png,Normal/masks/Normal-813.png,True,G17529,train,44172caeeeb6d7bd3258bb58d5126f91,d03cc22bbe693cd8
Normal-8130.png,Normal,Normal/images/Normal-8130.png,Normal/masks/Normal-8130.png,True,G17530,train,72628fb7bffa058747cdb2f45aa3f602,d96c861b154f194f
Normal-8131.png,Normal,Normal/images/Normal-8131.png,Normal/masks/Normal-8131.png,True,G17531,train,0e1ef0038b746e63acf891a3b986d6c6,d92906971b145f5e
Normal-8132.png,Normal,Normal/images/Normal-8132.png,Normal/masks/Normal-8132.png,True,G17532,train,b9a50231e7e41d66fc6e126d5534c535,d43d430e0c6c7dce
Normal-8133.png,Normal,Normal/images/Normal-8133.png,Normal/masks/Normal-8133.png,True,G17533,train,3f9279d8bc10033fcd4203b5a9c54835,dd2ed0431b8dc03f
Normal-8134.png,Normal,Normal/images/Normal-8134.png,Normal/masks/Normal-8134.png,True,G17534,train,5d134197a84d96b2eb5a4d8c5b08a615,9568475a0b6b1f78
Normal-8135.png,Normal,Normal/images/Normal-8135.png,Normal/masks/Normal-8135.png,True,G17535,train,26473f0a80d204a4cd3e0d55ff74d5d5,dd0ad30b0c1f0f4f
Normal-8136.png,Normal,Normal/images/Normal-8136.png,Normal/masks/Normal-8136.png,True,G17536,train,a8f8f166f66552a4f712fcfa59ef44fa,8169c61e185f7bb1
Normal-8137.png,Normal,Normal/images/Normal-8137.png,Normal/masks/Normal-8137.png,True,G17537,train,999fbc217c17ab4dcd42ea1c67128830,d92dd6061f174a1d
Normal-8138.png,Normal,Normal/images/Normal-8138.png,Normal/masks/Normal-8138.png,True,G17538,val,e440c8259baf59fc655012e645d477a5,d57c0a0b0f0f1f0f
Normal-8139.png,Normal,Normal/images/Normal-8139.png,Normal/masks/Normal-8139.png,True,G17539,test,9618d69126db199cf013d051ae9a0105,dd28d7570a4c4e5c
Normal-814.png,Normal,Normal/images/Normal-814.png,Normal/masks/Normal-814.png,True,G17540,train,8308ae4c89b50001112ae8e10020d18a,dcfc861f8f0d282c
Normal-8140.png,Normal,Normal/images/Normal-8140.png,Normal/masks/Normal-8140.png,True,G17541,val,c520681797d82a50b21c5c0da363a41d,9d3c071b151f5e45
Normal-8141.png,Normal,Normal/images/Normal-8141.png,Normal/masks/Normal-8141.png,True,G17542,test,836c9c67dd371d20c8dd7e6b399cc375,c03b5a3a3db4e466
Normal-8142.png,Normal,Normal/images/Normal-8142.png,Normal/masks/Normal-8142.png,True,G17543,val,9a126ce7f64aeee262fad9b57b8878ad,991fc71b18e939b0
Normal-8143.png,Normal,Normal/images/Normal-8143.png,Normal/masks/Normal-8143.png,True,G17544,train,9ad4e71256721f5135ddb592bb1178e7,9979871a184d5bb9
Normal-8144.png,Normal,Normal/images/Normal-8144.png,Normal/masks/Normal-8144.png,True,G17545,train,7e5e11113e5193dacefb92810919b29c,9568245e7ac96ec6
Normal-8145.png,Normal,Normal/images/Normal-8145.png,Normal/masks/Normal-8145.png,True,G17546,train,ea93cf9562c8ffa6c2770d2c5baa5138,c13916051e5f6f3a
Normal-8146.png,Normal,Normal/images/Normal-8146.png,Normal/masks/Normal-8146.png,True,G17547,test,f6a33733e637bdf35d5879b77de26c35,95698686131fd67a
Normal-8147.png,Normal,Normal/images/Normal-8147.png,Normal/masks/Normal-8147.png,True,G17548,val,c82f1a55bccb8418fc3639da3afa2849,d1bf421c560f1a4f
Normal-8148.png,Normal,Normal/images/Normal-8148.png,Normal/masks/Normal-8148.png,True,G17549,train,98f1ba336580b26ad4893138c8a44e1b,85687a967aa5c639
Normal-8149.png,Normal,Normal/images/Normal-8149.png,Normal/masks/Normal-8149.png,True,G17550,train,abadc9370fce6028bbf015f1c61cca87,d03d430a1f0b6f3d
Normal-815.png,Normal,Normal/images/Normal-815.png,Normal/masks/Normal-815.png,True,G17551,train,f2d5445ebe19977d61f6f5c45e1c1c4a,986f271a39c57cc4
Normal-8150.png,Normal,Normal/images/Normal-8150.png,Normal/masks/Normal-8150.png,True,G17552,train,fc7cce9ad4d8b95eec39629e98c306c4,d12916561b396de5
Normal-8151.png,Normal,Normal/images/Normal-8151.png,Normal/masks/Normal-8151.png,True,G17553,train,9f99edd480964fd3c6df1209629702e6,dd3ec24b1c490f9c
Normal-8152.png,Normal,Normal/images/Normal-8152.png,Normal/masks/Normal-8152.png,True,G17554,train,69766c4aef41ea8a3e2f75391ca10737,cc3ac20d564f1fe2
Normal-8153.png,Normal,Normal/images/Normal-8153.png,Normal/masks/Normal-8153.png,True,G17555,train,d59857db16df1986ba67339a3a61d6ad,d13c4fc61a0bac6b
Normal-8154.png,Normal,Normal/images/Normal-8154.png,Normal/masks/Normal-8154.png,True,G17556,train,69a1a113ed0a88c387ee74fd8a1a58f8,d82e471f165b784a
Normal-8155.png,Normal,Normal/images/Normal-8155.png,Normal/masks/Normal-8155.png,True,G17557,test,2377877f13a753b0ce10da23a41b4b41,8c4cc34b4d4e1eaf
Normal-8156.png,Normal,Normal/images/Normal-8156.png,Normal/masks/Normal-8156.png,True,G17558,train,115697ffb0b8a205f5769e7977c1978c,8169e7861a97475e
Normal-8157.png,Normal,Normal/images/Normal-8157.png,Normal/masks/Normal-8157.png,True,G17559,train,675fa9cfd46a66f896cd26bd6e605221,91691e1119dfe9e1
Normal-8158.png,Normal,Normal/images/Normal-8158.png,Normal/masks/Normal-8158.png,True,G17560,train,a1dd1c53d2b0ed5e2547e0383851856e,dc2e425f0a0e2fbc
Normal-8159.png,Normal,Normal/images/Normal-8159.png,Normal/masks/Normal-8159.png,True,G17561,train,c6aacf954117846fea594866d187e036,dc3d02093de869f5
Normal-816.png,Normal,Normal/images/Normal-816.png,Normal/masks/Normal-816.png,True,G17562,train,94c16c5cd88e09918ea18b2623afc89b,d565c74be9c568c0
Normal-8160.png,Normal,Normal/images/Normal-8160.png,Normal/masks/Normal-8160.png,True,G17563,train,cc1fefcc741fad197e8ae407e38b6068,d42f4a0b0f2f2d3c
Normal-8161.png,Normal,Normal/images/Normal-8161.png,Normal/masks/Normal-8161.png,True,G17564,val,bf203599aaf49cfb79ed390b2afbb51e,cc7b530f1b0d28ad
Normal-8162.png,Normal,Normal/images/Normal-8162.png,Normal/masks/Normal-8162.png,True,G17565,val,76d4c2071e1950996cbf10d4f91aa961,d42d43431f0e6f69
Normal-8163.png,Normal,Normal/images/Normal-8163.png,Normal/masks/Normal-8163.png,True,G17566,test,220db424c44dcb5684237cf555cc6339,9169961e1a5ee939
Normal-8164.png,Normal,Normal/images/Normal-8164.png,Normal/masks/Normal-8164.png,True,G17567,test,05c43d06af8b9c39c4ce68167cccacd6,d97dc6d61fc50818
Normal-8165.png,Normal,Normal/images/Normal-8165.png,Normal/masks/Normal-8165.png,True,G17568,train,94a7742f3a5f0b57317aa84d3fe02555,917a4707141e5b5f
Normal-8166.png,Normal,Normal/images/Normal-8166.png,Normal/masks/Normal-8166.png,True,G17569,train,61901a867421c844e2051fd2bb99e637,9078864f1b1b4f4f
Normal-8167.png,Normal,Normal/images/Normal-8167.png,Normal/masks/Normal-8167.png,True,G17570,train,903030f9c4b0d40952eac70de74cc545,dd0a470d6c7ce0f2
Normal-8168.png,Normal,Normal/images/Normal-8168.png,Normal/masks/Normal-8168.png,True,G17571,train,4893e6c4173bd9f740e8d5409994ce9a,9978861a1f5e185f
Normal-8169.png,Normal,Normal/images/Normal-8169.png,Normal/masks/Normal-8169.png,True,G17572,train,e7996a0b0d5f3f33e8405ea33b990c5f,dc35c30f4b2d3e24
Normal-817.png,Normal,Normal/images/Normal-817.png,Normal/masks/Normal-817.png,True,G17573,train,afe5ddfc8aa54b115eedd6c5d18ea439,dd7a425f5a0d3c0c
Normal-8170.png,Normal,Normal/images/Normal-8170.png,Normal/masks/Normal-8170.png,True,G17574,train,a208102c277587c09d4061e0922ac9ea,dc2a430f4c4e5c6f
Normal-8171.png,Normal,Normal/images/Normal-8171.png,Normal/masks/Normal-8171.png,True,G17575,train,b97c7cdb1e9261a8f1ac2603ec47c4a1,d939061c567cf869
Normal-8172.png,Normal,Normal/images/Normal-8172.png,Normal/masks/Normal-8172.png,True,G17576,train,8c12a0c1cdec4d9110b81bad9d57cb63,983d061a59697ff0
Normal-8173.png,Normal,Normal/images/Normal-8173.png,Normal/masks/Normal-8173.png,True,G17577,train,2f54648ec4b1714552fef2244df48ed0,887d86971a075e5b
Normal-8174.png,Normal,Normal/images/Normal-8174.png,Normal/masks/Normal-8174.png,True,G17578,val,862b6c3937cfebb66ed896b6cb0a2280,8960861e975f5e17
Normal-8175.png,Normal,Normal/images/Normal-8175.png,Normal/masks/Normal-8175.png,True,G17579,train,c2761f25825db4f8bf341b9bcc812f38,c13847131f4f3f19
Normal-8176.png,Normal,Normal/images/Normal-8176.png,Normal/masks/Normal-8176.png,True,G17580,train,1f7f13de60e90be2116e74147cd21fae,9979075a596d39a1
Normal-8177.png,Normal,Normal/images/Normal-8177.png,Normal/masks/Normal-8177.png,True,G17581,test,5ac67a3bf7b68bb6097371091aa918bf,9538da5a71da8627
Normal-8178.png,Normal,Normal/images/Normal-8178.png,Normal/masks/Normal-8178.png,True,G17582,test,35e1d60b74f900e87386f937cc404bd0,99211e1f1b4e78f1
Normal-8179.png,Normal,Normal/images/Normal-8179.png,Normal/masks/Normal-8179.png,True,G17583,train,166904ae4769b7631a7048ce24fa81f3,dc2dc2d70b4c4f4c
Normal-818.png,Normal,Normal/images/Normal-818.png,Normal/masks/Normal-818.png,True,G17573,train,afe5ddfc8aa54b115eedd6c5d18ea439,dd7a425f5a0d3c0c
Normal-8180.png,Normal,Normal/images/Normal-8180.png,Normal/masks/Normal-8180.png,True,G17584,train,d794a7ca82267c9a0f27b2e7a7a6c34d,dd3dc2191c4e396c
Normal-8181.png,Normal,Normal/images/Normal-8181.png,Normal/masks/Normal-8181.png,True,G17585,train,c03ed6c5c45d90cc0e6e4669a187bb35,987b871e5a4c5b58
Normal-8182.png,Normal,Normal/images/Normal-8182.png,Normal/masks/Normal-8182.png,True,G17586,val,d68598c95a62e37ab3b41c494097116a,91699e19134e797a
Normal-8183.png,Normal,Normal/images/Normal-8183.png,Normal/masks/Normal-8183.png,True,G17587,val,71e208c33d553c08c7e6df092fa8cbd1,9869c5165bc55f19
Normal-8184.png,Normal,Normal/images/Normal-8184.png,Normal/masks/Normal-8184.png,True,G17588,test,9eb98d81b1018ecb354c4cb202ddafa4,9169161f5a7e2939
Normal-8185.png,Normal,Normal/images/Normal-8185.png,Normal/masks/Normal-8185.png,True,G17589,train,076c2fc7288efbac294414bd36496b3d,dd2a530f4e0d3d2c
Normal-8186.png,Normal,Normal/images/Normal-8186.png,Normal/masks/Normal-8186.png,True,G17590,train,9a3a91186c4d8f8011daabfd134a978a,99af061c071f455f
Normal-8187.png,Normal,Normal/images/Normal-8187.png,Normal/masks/Normal-8187.png,True,G17591,train,11b577846e3c0b17bb86620297435ad2,9168071a1f5f783b
Normal-8188.png,Normal,Normal/images/Normal-8188.png,Normal/masks/Normal-8188.png,True,G17592,train,413d92667f7e49b8de59af92f4e58279,d12b46171e1d4b3b
Normal-8189.png,Normal,Normal/images/Normal-8189.png,Normal/masks/Normal-8189.png,True,G17593,train,e20c029fe13182b7718f819a126fc7bd,d42d4a4c0cecfcf4
Normal-819.png,Normal,Normal/images/Normal-819.png,Normal/masks/Normal-819.png,True,G17594,train,15d9d48689e1bcafb34459590e8beff9,c05ec31badc678d4
Normal-8190.png,Normal,Normal/images/Normal-8190.png,Normal/masks/Normal-8190.png,True,G17595,train,1e12da3ca5a220ea794e32286aca7ea6,d93d861f1d4c4b19
Normal-8191.png,Normal,Normal/images/Normal-8191.png,Normal/masks/Normal-8191.png,True,G17596,train,622d69470197ab811a3b18f550ff4a91,cd3cd257d20bbe40
Normal-8192.png,Normal,Normal/images/Normal-8192.png,Normal/masks/Normal-8192.png,True,G17597,train,5d46bcb2714c97616cc49a1d190bdffb,cc3c861b5c691e7c
Normal-8193.png,Normal,Normal/images/Normal-8193.png,Normal/masks/Normal-8193.png,True,G17598,train,c6b5515753109ff98b4deae0207592c8,843e0355386c7b7b
Normal-8194.png,Normal,Normal/images/Normal-8194.png,Normal/masks/Normal-8194.png,True,G17599,train,76826f22ed86b982a96cc3a748722cfc,997cc65b054f1b85
Normal-8195.png,Normal,Normal/images/Normal-8195.png,Normal/masks/Normal-8195.png,True,G17600,train,745bd077377ac64c9f4d371a512603df,dd2ac0531f4d4e1d
Normal-8196.png,Normal,Normal/images/Normal-8196.png,Normal/masks/Normal-8196.png,True,G17601,train,e4d4749daa7a316bf142de071220200e,dc2d42430fca4f3d
Normal-8197.png,Normal,Normal/images/Normal-8197.png,Normal/masks/Normal-8197.png,True,G17602,val,7d9174dc164b423ab91a1fe993d4124e,dd2e424b1fc40f3c
Normal-8198.png,Normal,Normal/images/Normal-8198.png,Normal/masks/Normal-8198.png,True,G17603,val,47834b2a8c764864fc547d19e218074f,d13d02865b1d574f
Normal-8199.png,Normal,Normal/images/Normal-8199.png,Normal/masks/Normal-8199.png,True,G17604,test,c1290134a8de350d3cf34481d8999283,817f06185d7c59ea
Normal-82.png,Normal,Normal/images/Normal-82.png,Normal/masks/Normal-82.png,True,G17605,test,6ce6e376fe5bd9ad9a4c3ee1ce21a116,d0b2e73bac2c38e4
Normal-820.png,Normal,Normal/images/Normal-820.png,Normal/masks/Normal-820.png,True,G17606,test,5bde1184d09ad1238e63f6c44010d4f6,d02de31e9c491dd9
Normal-8200.png,Normal,Normal/images/Normal-8200.png,Normal/masks/Normal-8200.png,True,G17607,train,a0ec01826d4974117a7633f9cb642413,992fc61a1dc41abd
Normal-8201.png,Normal,Normal/images/Normal-8201.png,Normal/masks/Normal-8201.png,True,G17608,train,dd18f0b49f9c3ee70c398e1831d1ade1,d42dd35f4a0d0e4d
Normal-8202.png,Normal,Normal/images/Normal-8202.png,Normal/masks/Normal-8202.png,True,G17609,val,a480320446ec38ff19257f3de746ec85,8969065adf7ae0e8
Normal-8203.png,Normal,Normal/images/Normal-8203.png,Normal/masks/Normal-8203.png,True,G17610,train,a6f0c6ed0e0eb93ff5c30119bf461d14,987b0796127b7929
Normal-8204.png,Normal,Normal/images/Normal-8204.png,Normal/masks/Normal-8204.png,True,G17611,train,1d21260e2a9739f35ea414ff8e7f62c1,9939c7501a475f1d
Normal-8205.png,Normal,Normal/images/Normal-8205.png,Normal/masks/Normal-8205.png,True,G17612,train,ed5129b89e181e5095349bb1d043baf6,987bc51616c15f1d
Normal-8206.png,Normal,Normal/images/Normal-8206.png,Normal/masks/Normal-8206.png,True,G17613,train,c954274adb187a9d80f18075e0b7e50c,d12f861e1d0e1e4f
Normal-8207.png,Normal,Normal/images/Normal-8207.png,Normal/masks/Normal-8207.png,True,G17614,train,6154a3660fbd579f4b37845fe02a51cb,d92dc61a154b3f58
Normal-8208.png,Normal,Normal/images/Normal-8208.png,Normal/masks/Normal-8208.png,True,G17615,val,c7ef21c43b3496bb34704228aa09ba9e,d92d465a154f1d4b
Normal-8209.png,Normal,Normal/images/Normal-8209.png,Normal/masks/Normal-8209.png,True,G17616,train,9061da554cfae517b1555bce85d5b008,817f061e1a5e7a78
Normal-821.png,Normal,Normal/images/Normal-821.png,Normal/masks/Normal-821.png,True,G17617,train,c6b87bf243fbbf910ade7d49a5f4de21,986bce5ad95919c4
Normal-8210.png,Normal,Normal/images/Normal-8210.png,Normal/masks/Normal-8210.png,True,G17618,train,f554546b610a8ac1e2aabba206a82dbf,997a051e1b4d3aab
Normal-8211.png,Normal,Normal/images/Normal-8211.png,Normal/masks/Normal-8211.png,True,G17619,train,cecdb13c35b1195ea903b659daa87c45,83601f1e1a5f3b3c
Normal-8212.png,Normal,Normal/images/Normal-8212.png,Normal/masks/Normal-8212.png,True,G17620,train,f9461c2b91f512c5c4f86d8929f382de,997f861e1c1a195b
Normal-8213.png,Normal,Normal/images/Normal-8213.png,Normal/masks/Normal-8213.png,True,G17621,train,f642a005cb5c354efafc2d48114dc6e4,997b47165a0d1b39
Normal-8214.png,Normal,Normal/images/Normal-8214.png,Normal/masks/Normal-8214.png,True,G17622,train,135745987662f2241e2d10e349284eae,c47fc30c2c573e07
Normal-8215.png,Normal,Normal/images/Normal-8215.png,Normal/masks/Normal-8215.png,True,G17623,val,f62225abdaf51f311463c5aba330bf2e,9918961e1d1d1d7d
Normal-8216.png,Normal,Normal/images/Normal-8216.png,Normal/masks/Normal-8216.png,True,G17624,val,8fe643fd093244b2a37f284b95db1a8a,dd2cd28b1f5c4a0d
Normal-8217.png,Normal,Normal/images/Normal-8217.png,Normal/masks/Normal-8217.png,True,G17625,test,ce4a644b5efb4d7841ffa5d30b14d673,f939c4c61b4d4b1c
Normal-8218.png,Normal,Normal/images/Normal-8218.png,Normal/masks/Normal-8218.png,True,G17626,test,780da9439a9ae775843ec557c4ffdfec,9179c6131a5daf58
Normal-8219.png,Normal,Normal/images/Normal-8219.png,Normal/masks/Normal-8219.png,True,G17627,train,5100aa2cf234d4bf4d7f2f181fd3440b,9761189c7bd96992
Normal-822.png,Normal,Normal/images/Normal-822.png,Normal/masks/Normal-822.png,True,G17628,test,f04fac66fb4da93782322830ca550707,802dce3ad9493dce
Normal-8220.png,Normal,Normal/images/Normal-8220.png,Normal/masks/Normal-8220.png,True,G17629,train,4634db58e06e3fce3ea92b7f38944833,816886841a9ff9bf
Normal-8221.png,Normal,Normal/images/Normal-8221.png,Normal/masks/Normal-8221.png,True,G17630,train,eb8bdabe09488f5bc29bf083e5eb4c35,dd2c464b186dec74
Normal-8222.png,Normal,Normal/images/Normal-8222.png,Normal/masks/Normal-8222.png,True,G17631,train,215676b5e94d6de851312e06d0728414,b975c61ac45a514f
Normal-8223.png,Normal,Normal/images/Normal-8223.png,Normal/masks/Normal-8223.png,True,G17632,train,8f62c27aabefbe1a4cb1706651d62cb3,d97b15861e981b99
Normal-8224.png,Normal,Normal/images/Normal-8224.png,Normal/masks/Normal-8224.png,True,G17633,train,d25216b8752eff78a6e0829a710a2d72,d979c40a5f457a15
Normal-8225.png,Normal,Normal/images/Normal-8225.png,Normal/masks/Normal-8225.png,True,G17634,train,c32037e5bb8189f5636ba5fbf8f2699c,dd2ac31b0c6c39ec
Normal-8226.png,Normal,Normal/images/Normal-8226.png,Normal/masks/Normal-8226.png,True,G17635,train,56625ebde73c911499eff69e3a22237a,8968065e78cd6af9
Normal-8227.png,Normal,Normal/images/Normal-8227.png,Normal/masks/Normal-8227.png,True,G17636,train,375c5c70cc2da6d88d345fce15f2964f,d12b57481a7b2a2f
Normal-8228.png,Normal,Normal/images/Normal-8228.png,Normal/masks/Normal-8228.png,True,G17637,val,5862b68f299841fe18ffefbce77aae7c,d9794718184f3bb8
Normal-8229.png,Normal,Normal/images/Normal-8229.png,Normal/masks/Normal-8229.png,True,G17638,train,9633cfbd78739e2b201d83cc58f808a2,cc3e42db4d5c4b4a
Normal-823.png,Normal,Normal/images/Normal-823.png,Normal/masks/Normal-823.png,True,G17639,test,6cb118fae3dbb5154bc9a064ad4c6490,916fc71ac94d19c6
Normal-8230.png,Normal,Normal/images/Normal-8230.png,Normal/masks/Normal-8230.png,True,G17640,train,8fa0384764598828444021e9e5bb37d2,d97b461e4a0f192b
Normal-8231.png,Normal,Normal/images/Normal-8231.png,Normal/masks/Normal-8231.png,True,G17641,train,417cab9fcc498c73b98661dc737ee794,897b861e185d5b39
Normal-8232.png,Normal,Normal/images/Normal-8232.png,Normal/masks/Normal-8232.png,True,G17642,train,5ccadf38a7e65f7e6adfd38180fc4a34,897f460b1e6d2c9c
Normal-8233.png,Normal,Normal/images/Normal-8233.png,Normal/masks/Normal-8233.png,True,G17643,val,c36031497d52f7250a4041364fb53f7f,9979851e5a0c5b79
Normal-8234.png,Normal,Normal/images/Normal-8234.png,Normal/masks/Normal-8234.png,True,G17644,val,d5d74eb68e0803d3eae3a6167015db87,99780618dd69b9e5
Normal-8235.png,Normal,Normal/images/Normal-8235.png,Normal/masks/Normal-8235.png,True,G17645,test,a26ecc33cfd2b5897f4402ce49c4dc29,8079a90fdfa958a9
Normal-8236.png,Normal,Normal/images/Normal-8236.png,Normal/masks/Normal-8236.png,True,G17646,train,aaf68a04c99bd2aa6056c1954c828caa,997985121f0f5a1f
Normal-8237.png,Normal,Normal/images/Normal-8237.png,Normal/masks/Normal-8237.png,True,G17647,train,daf0fdf8b72c21f1de6de915cda1fd3a,dd3e434b094d0e3d
Normal-8238.png,Normal,Normal/images/Normal-8238.png,Normal/masks/Normal-8238.png,True,G17648,train,876c402171d8468213fc22ab8dd25ba8,817b56065e1b1b1f
Normal-8239.png,Normal,Normal/images/Normal-8239.png,Normal/masks/Normal-8239.png,True,G17649,test,51c2c6c317f9b589596af5d60ca84ee6,9568061b0e5d5f4f
Normal-824.png,Normal,Normal/images/Normal-824.png,Normal/masks/Normal-824.png,True,G17650,train,e233b1112b6684e202f3b614ba3cf8e0,903dc61abcc93dd4
Normal-8240.png,Normal,Normal/images/Normal-8240.png,Normal/masks/Normal-8240.png,True,G17651,train,3cb6300b396259107ad89482ba525ce4,d939c41e1e561f1c
Normal-8241.png,Normal,Normal/images/Normal-8241.png,Normal/masks/Normal-8241.png,True,G17652,train,eb747f9be3fd027a85637c5db84cd692,ec2c831b051f1f4f
Normal-8242.png,Normal,Normal/images/Normal-8242.png,Normal/masks/Normal-8242.png,True,G17653,train,5d80237e60f60ba8cce586b9b0ae5871,dd2956c25a1e4e4b
Normal-8243.png,Normal,Normal/images/Normal-8243.png,Normal/masks/Normal-8243.png,True,G17654,test,029745641d62fa65042228f28b15cd08,cc3d425b1f4e4c39
Normal-8244.png,Normal,Normal/images/Normal-8244.png,Normal/masks/Normal-8244.png,True,G17655,train,35cd29672725e667f00ea2d644ca0958,dd2cc4021d4d5f1f
Normal-8245.png,Normal,Normal/images/Normal-8245.png,Normal/masks/Normal-8245.png,True,G17656,train,7ad58cfd7d81fad4dc7ade9534ee47e6,d978d71e544b5948
Normal-8246.png,Normal,Normal/images/Normal-8246.png,Normal/masks/Normal-8246.png,True,G17657,train,84537135d8afccbcf3063271c0bfd54f,d93dc61311455f1e
Normal-8247.png,Normal,Normal/images/Normal-8247.png,Normal/masks/Normal-8247.png,True,G17658,train,49718caadba55734b90447e684ea39af,81695e86581fcd7a
Normal-8248.png,Normal,Normal/images/Normal-8248.png,Normal/masks/Normal-8248.png,True,G17659,test,a510c4bee943bd12927658b774eb521d,887dc7de184d4e38
Normal-8249.png,Normal,Normal/images/Normal-8249.png,Normal/masks/Normal-8249.png,True,G17660,train,181c0a3ba2ca01b32cc4f26383cd300e,dd39c21f184b3d70
Normal-825.png,Normal,Normal/images/Normal-825.png,Normal/masks/Normal-825.png,True,G17661,train,cf2a559561ad2bd3f1b5a3956e541e6d,911dc31af8b53cb4
Normal-8250.png,Normal,Normal/images/Normal-8250.png,Normal/masks/Normal-8250.png,True,G17662,train,4fb3f92504b7d73f52520e1d658efc9f,9c2c439b554f5d49
Normal-8251.png,Normal,Normal/images/Normal-8251.png,Normal/masks/Normal-8251.png,True,G17663,test,9f06ff7a8c40c5848f1fb4bc871d7d6d,993b071e9e5e1a1a
Normal-8252.png,Normal,Normal/images/Normal-8252.png,Normal/masks/Normal-8252.png,True,G17664,train,71e97df89f7a45404ff26cc074aad1d2,9979871612875f1b
Normal-8253.png,Normal,Normal/images/Normal-8253.png,Normal/masks/Normal-8253.png,True,G17665,val,2bd4964b7b7e2422b7c7849b8ddc4aee,996d871a155e78c5
Normal-8254.png,Normal,Normal/images/Normal-8254.png,Normal/masks/Normal-8254.png,True,G17666,val,83c3d61239ea7f8b5f306ddb0b0e0e40,d03d4a5d5d483d3c
Normal-8255.png,Normal,Normal/images/Normal-8255.png,Normal/masks/Normal-8255.png,True,G17667,val,4fd9118cc57e9944332b1c1f465f9259,9178c6461b0faf3a
Normal-8256.png,Normal,Normal/images/Normal-8256.png,Normal/masks/Normal-8256.png,True,G17668,train,d757137953c0ac19ce00535d39687346,dc3d420f0a6a3dbc
Normal-8257.png,Normal,Normal/images/Normal-8257.png,Normal/masks/Normal-8257.png,True,G17669,train,c1a098571863c61a73c255078d99c4e0,d920465a1acf3b9d
Normal-8258.png,Normal,Normal/images/Normal-8258.png,Normal/masks/Normal-8258.png,True,G17670,train,ddf8999ef9bcee4ea883f43190c4ca52,917917061a5de979
Normal-8259.png,Normal,Normal/images/Normal-8259.png,Normal/masks/Normal-8259.png,True,G17671,train,be8312474277b1c8d74c2273052e53ba,990f16586cf869e9
Normal-826.png,Normal,Normal/images/Normal-826.png,Normal/masks/Normal-826.png,True,G17672,train,c177b1085f5628bde1a5edbd425b8f5a,dc2e034be5acbcc4
Normal-8260.png,Normal,Normal/images/Normal-8260.png,Normal/masks/Normal-8260.png,True,G17673,train,4a91ac4ffa12af67636296cf02e0e874,c52e061f1c4f5e6a
Normal-8261.png,Normal,Normal/images/Normal-8261.png,Normal/masks/Normal-8261.png,True,G17674,train,29ea4f5d6c5666c83c1fb11dce54d4dd,8879c79e58c15f1a
Normal-8262.png,Normal,Normal/images/Normal-8262.png,Normal/masks/Normal-8262.png,True,G17675,train,eced9ac66623e22e70a26c1dfdf574b6,98740e0f191fee39
Normal-8263.png,Normal,Normal/images/Normal-8263.png,Normal/masks/Normal-8263.png,True,G17676,train,20937effdba99077527ffef9d9bc2cfd,d52a430b3f2d3536
Normal-8264.png,Normal,Normal/images/Normal-8264.png,Normal/masks/Normal-8264.png,True,G17677,train,c0dce22094e8132ee0f242cfe6d61e39,983d87061a4f1f5e
Normal-8265.png,Normal,Normal/images/Normal-8265.png,Normal/masks/Normal-8265.png,True,G17678,train,70a859baae9d3a999e43f1bac384ade7,c83d421d1e4a7cf9
Normal-8266.png,Normal,Normal/images/Normal-8266.png,Normal/masks/Normal-8266.png,True,G17679,train,2cbdb52c1679394f3c4a653b8a934ff6,9179561bf9e5a190
Normal-8267.png,Normal,Normal/images/Normal-8267.png,Normal/masks/Normal-8267.png,True,G17680,train,083b6b16f43dbbb11ce03a1280186691,d43d434d0d2e3cbc
Normal-8268.png,Normal,Normal/images/Normal-8268.png,Normal/masks/Normal-8268.png,True,G17681,train,af8351a56b9fcd5b6e9a5905b1f39dc9,987d871e415f1b49
Normal-8269.png,Normal,Normal/images/Normal-8269.png,Normal/masks/Normal-8269.png,True,G17682,val,012c639adb97bf7abf93345874ed62cf,8c0c474f4c68f8ef
Normal-827.png,Normal,Normal/images/Normal-827.png,Normal/masks/Normal-827.png,True,G17683,val,a87f61e150c6e231f4c4e072db10f722,9929c63ad86d7949
Normal-8270.png,Normal,Normal/images/Normal-8270.png,Normal/masks/Normal-8270.png,True,G17684,val,128d78f4d0cfc758eb14f102c28ec305,817b06161a5f7a5b
Normal-8271.png,Normal,Normal/images/Normal-8271.png,Normal/masks/Normal-8271.png,True,G17685,test,c44fc3eb71bedf17773f7c4c414154f2,993f85121bc45f59
Normal-8272.png,Normal,Normal/images/Normal-8272.png,Normal/masks/Normal-8272.png,True,G17686,test,4df949fc5da115bb86a918491b52c06c,c93d06461f1a4f3d
Normal-8273.png,Normal,Normal/images/Normal-8273.png,Normal/masks/Normal-8273.png,True,G17687,test,c91ce987d8bf083c849fe108f4aab5e8,d83f521d5e4f190a
Normal-8274.png,Normal,Normal/images/Normal-8274.png,Normal/masks/Normal-8274.png,True,G17688,train,717d7fb9ffcb4b6146d3004e0f660dd9,912f0617165e59dd
Normal-8275.png,Normal,Normal/images/Normal-8275.png,Normal/masks/Normal-8275.png,True,G17689,train,3e4de8d90bd86965d04f42df865b0300,987987169adf3a18
Normal-8276.png,Normal,Normal/images/Normal-8276.png,Normal/masks/Normal-8276.png,True,G17690,train,50d1a691c9b627fe003e75426c85b5ab,dd2c401b1decb8b6
Normal-8277.png,Normal,Normal/images/Normal-8277.png,Normal/masks/Normal-8277.png,True,G17691,train,23390ef4999889d1d0656b9efed4fe1f,d52e031a087e3e7d
Normal-8278.png,Normal,Normal/images/Normal-8278.png,Normal/masks/Normal-8278.png,True,G17692,train,73bf627bc9c67fa53b2d3ac12e56bd25,dd7cc21b51c71919
Normal-8279.png,Normal,Normal/images/Normal-8279.png,Normal/masks/Normal-8279.png,True,G17693,train,1a987b06bcb5adc743eab027d1a2ab20,a873c21a4a1b5b3f
Normal-828.png,Normal,Normal/images/Normal-828.png,Normal/masks/Normal-828.png,True,G17694,train,205c06ed845f58641fdfe105547d4122,956e0f1b0b5c7964
Normal-8280.png,Normal,Normal/images/Normal-8280.png,Normal/masks/Normal-8280.png,True,G17695,train,b522b5da39d7ac6b226fcb944a54820d,817ac75a58615b79
Normal-8281.png,Normal,Normal/images/Normal-8281.png,Normal/masks/Normal-8281.png,True,G17696,train,2229d7a3049b0bc62b19a01a144ce40f,897957061a1e4f1f
Normal-8282.png,Normal,Normal/images/Normal-8282.png,Normal/masks/Normal-8282.png,True,G17697,train,5e4346fcf417efae24f7832f9ae3715a,937884505b6d7bd8
Normal-8283.png,Normal,Normal/images/Normal-8283.png,Normal/masks/Normal-8283.png,True,G17698,train,8ce021843ac423dedaa8b19a9d4d4f1c,9979c71a94de1a94
Normal-8284.png,Normal,Normal/images/Normal-8284.png,Normal/masks/Normal-8284.png,True,G17699,test,6c832d7851fc97c34e13211db6e5f60b,9978571b1bc50d39
Normal-8285.png,Normal,Normal/images/Normal-8285.png,Normal/masks/Normal-8285.png,True,G17700,train,576275bcf4cb6a7e9bbed1e0f624e64b,997a861a194d5af5
Normal-8286.png,Normal,Normal/images/Normal-8286.png,Normal/masks/Normal-8286.png,True,G17701,train,8fcc7b819ab43e7148c3a9627b17b68c,cc0f434b4d6c49f3
Normal-8287.png,Normal,Normal/images/Normal-8287.png,Normal/masks/Normal-8287.png,True,G17702,train,9531567af2f1bc00eeaee4c089e4a400,d92d065f7d4848f8
Normal-8288.png,Normal,Normal/images/Normal-8288.png,Normal/masks/Normal-8288.png,True,G17703,val,6a3d52df943974bba7aad1fa309b7cc9,f86987929acd98b8
Normal-8289.png,Normal,Normal/images/Normal-8289.png,Normal/masks/Normal-8289.png,True,G17704,train,7dd8c76f035d739a67676263f9aca7e1,d42fc34b0a422f3f
Normal-829.png,Normal,Normal/images/Normal-829.png,Normal/masks/Normal-829.png,True,G17705,val,e46b6f9cd08f28596b7a671638c3a327,d17dc61a9b2b3864
Normal-8290.png,Normal,Normal/images/Normal-8290.png,Normal/masks/Normal-8290.png,True,G17706,val,53eacdce96f8114094e63aa79f9a7693,dc3dd24349d54b0d
Normal-8291.png,Normal,Normal/images/Normal-8291.png,Normal/masks/Normal-8291.png,True,G17707,test,eafc89091fb7694b80653d1bdcb6e03b,dd6c461b0f0e3a1b
Normal-8292.png,Normal,Normal/images/Normal-8292.png,Normal/masks/Normal-8292.png,True,G17708,test,8bf852eebcd3cce8dbfeebc095e39438,817a86141d5f1d5f
Normal-8293.png,Normal,Normal/images/Normal-8293.png,Normal/masks/Normal-8293.png,True,G17709,test,f2b43f0623a1f9b5a5fa21571a51ce1b,813b56861a1cafeb
Normal-8294.png,Normal,Normal/images/Normal-8294.png,Normal/masks/Normal-8294.png,True,G17710,val,b76174a4d7d8c44314c1eb13714f3255,d508cbcb0dacf5b0
Normal-8295.png,Normal,Normal/images/Normal-8295.png,Normal/masks/Normal-8295.png,True,G17711,train,ba0b0e94660d5b8ae8b37703268aee74,b939c6131f444f59
Normal-8296.png,Normal,Normal/images/Normal-8296.png,Normal/masks/Normal-8296.png,True,G17712,train,28ea16c3d41740aa12d161d131625107,882e575b4d4d689d
Normal-8297.png,Normal,Normal/images/Normal-8297.png,Normal/masks/Normal-8297.png,True,G17713,train,0f75bdaacd10c93e42d108a01128dd36,992d061a1d4f3b6d
Normal-8298.png,Normal,Normal/images/Normal-8298.png,Normal/masks/Normal-8298.png,True,G17714,train,da625d20ccc6489de039b1f7ef236039,dd2cd21b5b430f19
Normal-8299.png,Normal,Normal/images/Normal-8299.png,Normal/masks/Normal-8299.png,True,G17715,train,e80f3b1c0d7afa475bf010b10401e521,817886125b5f1e5f
Normal-83.png,Normal,Normal/images/Normal-83.png,Normal/masks/Normal-83.png,True,G17716,train,ebd03a452418e0d1629ef8884fc74b27,813dc618e8693dcf
Normal-830.png,Normal,Normal/images/Normal-830.png,Normal/masks/Normal-830.png,True,G17717,train,af0726bb31949540857c5c9334401ce8,d02ce23fed683cd0
Normal-8300.png,Normal,Normal/images/Normal-8300.png,Normal/masks/Normal-8300.png,True,G17718,train,fec38717230f2292d74e4074a9c0bb02,d57d821e1e0e1b4b
Normal-8301.png,Normal,Normal/images/Normal-8301.png,Normal/masks/Normal-8301.png,True,G17719,train,7cfbdca6f6f5ad350158f8cb9fa25817,993b879417861ed3
Normal-8302.png,Normal,Normal/images/Normal-8302.png,Normal/masks/Normal-8302.png,True,G17720,train,071d7a9ba79d532403f41c32bff11ef5,ed3cd24b1d4c4f0c
Normal-8303.png,Normal,Normal/images/Normal-8303.png,Normal/masks/Normal-8303.png,True,G17721,train,3025a07bc579428fd483ea7384380a34,957f871a154f1a0d
Normal-8304.png,Normal,Normal/images/Normal-8304.png,Normal/masks/Normal-8304.png,True,G17722,train,58141c8c3e407f8df884ce5ebe949bf9,8c3dd4521f475a1d
Normal-8305.png,Normal,Normal/images/Normal-8305.png,Normal/masks/Normal-8305.png,True,G17723,train,60fcd53ba44a8b265beb495a0dd97da9,897d061a8b3fb8a5
Normal-8306.png,Normal,Normal/images/Normal-8306.png,Normal/masks/Normal-8306.png,True,G17724,train,d1b918227c5b07840db0514f0cdb96e1,d43cc31948ceb8ed
Normal-8307.png,Normal,Normal/images/Normal-8307.png,Normal/masks/Normal-8307.png,True,G17725,train,4e9ee74819139597f11ea432abf1fc22,91680e171b6f6d78
Normal-8308.png,Normal,Normal/images/Normal-8308.png,Normal/masks/Normal-8308.png,True,G17726,test,c2c46489eb486b9603389f27efe39d70,cc3c43494c663df3
Normal-8309.png,Normal,Normal/images/Normal-8309.png,Normal/masks/Normal-8309.png,True,G17727,val,4372a97123aa6c885a223015a8e80c3b,817ec718595b195b
Normal-831.png,Normal,Normal/images/Normal-831.png,Normal/masks/Normal-831.png,True,G17728,val,f28242e84061d0be22c99a0118611651,983bd65a5a5b1859
Normal-8310.png,Normal,Normal/images/Normal-8310.png,Normal/masks/Normal-8310.png,True,G17729,test,e1c1359ae26c8fab42c3cddcd26de1da,8d3c07411a4f7bbc
Normal-8311.png,Normal,Normal/images/Normal-8311.png,Normal/masks/Normal-8311.png,True,G17730,train,cadcadcf9e05c1b876120d7cc6dd2c88,d93fc61a1d411f4c
Normal-8312.png,Normal,Normal/images/Normal-8312.png,Normal/masks/Normal-8312.png,True,G17731,val,0a45417335e671f19d2072398f8c35d7,81795616181edf7a
Normal-8313.png,Normal,Normal/images/Normal-8313.png,Normal/masks/Normal-8313.png,True,G17732,val,de714826f9fe6b6a6da9e577e6522ce4,89fbd618055a555b
Normal-8314.png,Normal,Normal/images/Normal-8314.png,Normal/masks/Normal-8314.png,True,G17733,train,bc57e07be12b5e9000b664a124e71825,dd2f460a1b0d2eb9
Normal-8315.png,Normal,Normal/images/Normal-8315.png,Normal/masks/Normal-8315.png,True,G17734,train,87aa6d2395c821702780062f82c08c66,d969c71e451e154b
Normal-8316.png,Normal,Normal/images/Normal-8316.png,Normal/masks/Normal-8316.png,True,G17735,train,614d475a8bfef3b632abf7e539d6e387,98996dd8d863ca8e
Normal-8317.png,Normal,Normal/images/Normal-8317.png,Normal/masks/Normal-8317.png,True,G17736,train,76e1c8c175f0aec999b238212c1f7c01,8178171a5bdd78c9
Normal-8318.png,Normal,Normal/images/Normal-8318.png,Normal/masks/Normal-8318.png,True,G17737,train,318d3db03e16e54a02db47195342eba9,9978ce4919472eb9
Normal-8319.png,Normal,Normal/images/Normal-8319.png,Normal/masks/Normal-8319.png,True,G17738,train,b0ae979a94f1eccc46020a2a75143ce4,c03b561d1a5f4d39
Normal-832.png,Normal,Normal/images/Normal-832.png,Normal/masks/Normal-832.png,True,G17739,train,b2f5532c41ee33914749c6023213512c,d47ea71fad0d2980
Normal-8320.png,Normal,Normal/images/Normal-8320.png,Normal/masks/Normal-8320.png,True,G17740,train,3519d3048a956457d7a2396fe5ceec0f,983f061e1e5a3a79
Normal-8321.png,Normal,Normal/images/Normal-8321.png,Normal/masks/Normal-8321.png,True,G17741,train,3732cdc7de34c5515ea4fb1b15a05d26,d42ec60b08ee6bf8
Normal-8322.png,Normal,Normal/images/Normal-8322.png,Normal/masks/Normal-8322.png,True,G17742,train,11a65a06598108b80fd0ced460d067af,887f061a1f5e1a5e
Normal-8323.png,Normal,Normal/images/Normal-8323.png,Normal/masks/Normal-8323.png,True,G17743,train,e0f8d1e70fa2882a301a690df06b0ee5,8178171a1bf979b1
Normal-8324.png,Normal,Normal/images/Normal-8324.png,Normal/masks/Normal-8324.png,True,G17744,val,0ea81fa79d0f335a54ce70020a496151,dd2c021b554b3c6f
Normal-8325.png,Normal,Normal/images/Normal-8325.png,Normal/masks/Normal-8325.png,True,G17745,test,2282671332a011112c03aa17be625dd6,d939c6471a174f1a
Normal-8326.png,Normal,Normal/images/Normal-8326.png,Normal/masks/Normal-8326.png,True,G17746,val,f690d85d5f883e8e070b2b109fdb412a,997f061e1959594b
Normal-8327.png,Normal,Normal/images/Normal-8327.png,Normal/masks/Normal-8327.png,True,G17747,test,c95d1c1eb8fd10d508a5c4181198c7a5,807be7d69e84c61c
Normal-8328.png,Normal,Normal/images/Normal-8328.png,Normal/masks/Normal-8328.png,True,G17748,val,d57b023fb09b0144edb597e43365b95e,896c065a0d5d4f5f
Normal-8329.png,Normal,Normal/images/Normal-8329.png,Normal/masks/Normal-8329.png,True,G17749,test,36f63832b4d9b1cbd36626c5c0db750d,97625a79616d90dc
Normal-833.png,Normal,Normal/images/Normal-833.png,Normal/masks/Normal-833.png,True,G17750,train,635b7275d515b88188228ca5684d569b,d83cc31e5f591c49
Normal-8330.png,Normal,Normal/images/Normal-8330.png,Normal/masks/Normal-8330.png,True,G17751,train,641ba94b7636748d66e14dde6ecc1b28,cc3ac34f1c6d2c6c
Normal-8331.png,Normal,Normal/images/Normal-8331.png,Normal/masks/Normal-8331.png,True,G17752,train,41c60be63da6483ea3290a8b3c96a337,917886061b5f6f1b
Normal-8332.png,Normal,Normal/images/Normal-8332.png,Normal/masks/Normal-8332.png,True,G17753,train,4fea18487f43c6f46552f74306bf3c21,9905061e5d5f7c4d
Normal-8333.png,Normal,Normal/images/Normal-8333.png,Normal/masks/Normal-8333.png,True,G17754,val,2616066266bd547beecfafa2f599a47a,f939c6160e861ece
Normal-8334.png,Normal,Normal/images/Normal-8334.png,Normal/masks/Normal-8334.png,True,G17755,train,ecfb9af2301c03c8fc0247a2d4c113c3,d03dd34b5ad2435a
Normal-8335.png,Normal,Normal/images/Normal-8335.png,Normal/masks/Normal-8335.png,True,G17756,train,a15c20d73cfdbccf0c3f18108e40da87,9979071a5d6938e9
Normal-8336.png,Normal,Normal/images/Normal-8336.png,Normal/masks/Normal-8336.png,True,G17757,train,c62b2d0c07efe64383341f4df53692c4,d93dc21f1c4e1b38
Normal-8337.png,Normal,Normal/images/Normal-8337.png,Normal/masks/Normal-8337.png,True,G17758,train,978dd8d4b527c1b3e96ba7f44e47ae2c,c02a575b58585b5f
Normal-8338.png,Normal,Normal/images/Normal-8338.png,Normal/masks/Normal-8338.png,True,G17759,train,c807f78068f6b27d9226eaa40af858bf,91b92e4239db69e8
Normal-8339.png,Normal,Normal/images/Normal-8339.png,Normal/masks/Normal-8339.png,True,G17760,train,4bf85254986b595fd950d94322670dd5,9939861e1c4e1f79
Normal-834.png,Normal,Normal/images/Normal-834.png,Normal/masks/Normal-834.png,True,G17761,train,76f52c332af10a944046aa21901191e2,d4f4c31b8d2d3c64
Normal-8340.png,Normal,Normal/images/Normal-8340.png,Normal/masks/Normal-8340.png,True,G17762,train,b7f8c74d1b482f8871e587ed1bf6f625,9c4f430b2c2e39e7
Normal-8341.png,Normal,Normal/images/Normal-8341.png,Normal/masks/Normal-8341.png,True,G17763,train,3d4bd5dd97e9002f961111e11566eb72,dc344b0b1a4f3f38
Normal-8342.png,Normal,Normal/images/Normal-8342.png,Normal/masks/Normal-8342.png,True,G17764,train,c6843ccc1321ab470901631fbcd2831d,d428570e4a3f6a6d
Normal-8343.png,Normal,Normal/images/Normal-8343.png,Normal/masks/Normal-8343.png,True,G17765,val,276fba126fc13092ca154e5e218687a0,d938065f13c74f13
Normal-8344.png,Normal,Normal/images/Normal-8344.png,Normal/masks/Normal-8344.png,True,G17766,test,7abf75038da0a366cdee318ad2cff699,c43dc34b4d442f2f
Normal-8345.png,Normal,Normal/images/Normal-8345.png,Normal/masks/Normal-8345.png,True,G17767,test,fffc1fdcbbd1bef4326c70f7cef004b8,993dc61b18863bbc
Normal-8346.png,Normal,Normal/images/Normal-8346.png,Normal/masks/Normal-8346.png,True,G17768,test,fd6daab405090cc029ba85d21674dc99,97601e5f5a79682a
Normal-8347.png,Normal,Normal/images/Normal-8347.png,Normal/masks/Normal-8347.png,True,G17769,train,cc623b5d39282a1e66b8caf257e9e2bf,997b865e5d18591c
Normal-8348.png,Normal,Normal/images/Normal-8348.png,Normal/masks/Normal-8348.png,True,G17770,train,4e64c240c6b0deca0cafe60f47de1169,cc3cd20b1f4a1d6d
Normal-8349.png,Normal,Normal/images/Normal-8349.png,Normal/masks/Normal-8349.png,True,G17771,train,8750fd027e10e589e374ba15bb424f22,cd3c421b1d68bd6c
Normal-835.png,Normal,Normal/images/Normal-835.png,Normal/masks/Normal-835.png,True,G17772,train,b0afa796d75a54e0f25477a857d6d7a1,80aace5fdd4f1845
Normal-8350.png,Normal,Normal/images/Normal-8350.png,Normal/masks/Normal-8350.png,True,G17773,train,0298c6eb1820526b389b5bacd515a522,dd2c421b0ba83dfc
Normal-8351.png,Normal,Normal/images/Normal-8351.png,Normal/masks/Normal-8351.png,True,G17774,val,034b053d4e0a5687954c00c53458ea73,856e065a5d68fe68
Normal-8352.png,Normal,Normal/images/Normal-8352.png,Normal/masks/Normal-8352.png,True,G17775,train,3f8766b0bd4d615d6eadf94bfd366850,9939065f191e4e79
Normal-8353.png,Normal,Normal/images/Normal-8353.png,Normal/masks/Normal-8353.png,True,G17776,val,e8ba3d563a4928448aded11d957059a1,d92902171e5c5f79
Normal-8354.png,Normal,Normal/images/Normal-8354.png,Normal/masks/Normal-8354.png,True,G17777,test,e7cce41c7cd270ed6c7cff4e10eaa9a8,91780746139eeb2f
Normal-8355.png,Normal,Normal/images/Normal-8355.png,Normal/masks/Normal-8355.png,True,G17778,train,e7b4e5370ca43dae04f62ddccc61f2de,997847837ac56cb8
Normal-8356.png,Normal,Normal/images/Normal-8356.png,Normal/masks/Normal-8356.png,True,G17779,train,b02e8fac872fe059adc9c56a1ee61788,d528175b185d4d79
Normal-8357.png,Normal,Normal/images/Normal-8357.png,Normal/masks/Normal-8357.png,True,G17780,train,903d00d5631b0b8f1fd2d40591258d6d,d93a564e0c2f2b2d
Normal-8358.png,Normal,Normal/images/Normal-8358.png,Normal/masks/Normal-8358.png,True,G17781,train,6940d851828eb6989be2163f50d9a15e,d93dc65315461b1d
Normal-8359.png,Normal,Normal/images/Normal-8359.png,Normal/masks/Normal-8359.png,True,G17782,train,3ef580ca2ea2bde9142be22fb7c0b321,897d871696463e95
Normal-836.png,Normal,Normal/images/Normal-836.png,Normal/masks/Normal-836.png,True,G17783,train,11ede42120273895f8392e643d6d2577,913bde1e18511d97
Normal-8360.png,Normal,Normal/images/Normal-8360.png,Normal/masks/Normal-8360.png,True,G17784,train,df2dccbd0ab53b3c3f501ea20a9c5d92,992d861a1acd3ef8
Normal-8361.png,Normal,Normal/images/Normal-8361.png,Normal/masks/Normal-8361.png,True,G17785,test,a589d1372b9ce4787828ec942cbb0561,9978c61f19446d79
Normal-8362.png,Normal,Normal/images/Normal-8362.png,Normal/masks/Normal-8362.png,True,G17786,train,aeae1c19b3185b6bf4706cbc74f9188b,dc2fc05b4f480e4f
Normal-8363.png,Normal,Normal/images/Normal-8363.png,Normal/masks/Normal-8363.png,True,G17787,test,9e40f78f7498c5419c9a3e553546feb5,d96fc61b0e1a390e
Normal-8364.png,Normal,Normal/images/Normal-8364.png,Normal/masks/Normal-8364.png,True,G17788,val,b0b4ddaf95fbf557382d3ae7e13d570a,9d3c60d33ac46f94
Normal-8365.png,Normal,Normal/images/Normal-8365.png,Normal/masks/Normal-8365.png,True,G17789,train,1c5af103d340ee1c2d2e5393cf41de0c,91688617915f5f1d
Normal-8366.png,Normal,Normal/images/Normal-8366.png,Normal/masks/Normal-8366.png,True,G17790,train,3eb1c509943f79b53ea36303d6d8ac54,8179061e781d7f39
Normal-8367.png,Normal,Normal/images/Normal-8367.png,Normal/masks/Normal-8367.png,True,G17791,train,6c7caeadfbd8f607a54451551072031b,dc1a532d1ca99ea3
Normal-8368.png,Normal,Normal/images/Normal-8368.png,Normal/masks/Normal-8368.png,True,G17792,train,1a82a43bca35bb76a5f798230cc40062,dc3c421f4a4f3cf0
Normal-8369.png,Normal,Normal/images/Normal-8369.png,Normal/masks/Normal-8369.png,True,G17793,train,c4fd55fdc419867a7190e56febbfe131,dcbe475b444f184c
Normal-837.png,Normal,Normal/images/Normal-837.png,Normal/masks/Normal-837.png,True,G17794,train,3530cf7a5ecb7df986b23a9ae9d90e62,9d6f870b890f4c4d
Normal-8370.png,Normal,Normal/images/Normal-8370.png,Normal/masks/Normal-8370.png,True,G17795,train,03032eb4e750648d52d6c0a752df8ae9,d03d06475819ed6f
Normal-8371.png,Normal,Normal/images/Normal-8371.png,Normal/masks/Normal-8371.png,True,G17796,train,333a5aac84f5083ed55879f899882efb,9978071a5e4d78e9
Normal-8372.png,Normal,Normal/images/Normal-8372.png,Normal/masks/Normal-8372.png,True,G17797,train,e69c4f68842d70327c80c0903b256302,8169de1e581fa85b
Normal-8373.png,Normal,Normal/images/Normal-8373.png,Normal/masks/Normal-8373.png,True,G17798,train,242a3cf550749ad7817c63af6f43511a,9879861e114f787d
Normal-8374.png,Normal,Normal/images/Normal-8374.png,Normal/masks/Normal-8374.png,True,G17799,train,17695f7803a0a0bda7133eed7940437f,d53dc6d61e4a4c1a
Normal-8375.png,Normal,Normal/images/Normal-8375.png,Normal/masks/Normal-8375.png,True,G17800,train,7cd52c0459a0eec197559b47a6d314bc,cc4dc34d4c0f0f2f
Normal-8376.png,Normal,Normal/images/Normal-8376.png,Normal/masks/Normal-8376.png,True,G17801,train,02e4ec5f51e5bd9a4d8c5ab9cb8336c7,8179161e5adf6839
Normal-8377.png,Normal,Normal/images/Normal-8377.png,Normal/masks/Normal-8377.png,True,G17802,test,2ad117684bc56f96a735b6a886da71db,d46f2c33683162b7
Normal-8378.png,Normal,Normal/images/Normal-8378.png,Normal/masks/Normal-8378.png,True,G17803,train,bdf691af1430514701eae45c5145aaf0,dd2ec20b5d4d0e4d
Normal-8379.png,Normal,Normal/images/Normal-8379.png,Normal/masks/Normal-8379.png,True,G17804,val,0d996e79b74ea5eee3baec9a92f36831,89e9c616844efe47
Normal-838.png,Normal,Normal/images/Normal-838.png,Normal/masks/Normal-838.png,True,G17805,train,f95c7df1ff3b033d730397b6169cba3f,d078425e0b8b2fbd
Normal-8380.png,Normal,Normal/images/Normal-8380.png,Normal/masks/Normal-8380.png,True,G17806,val,520e1fd0ada10f284ea1ebe9dc43cf3c,8c5f035b4d5e18ea
Normal-8381.png,Normal,Normal/images/Normal-8381.png,Normal/masks/Normal-8381.png,True,G17807,val,bdf66e1a4afba3d3e13108842571ff6f,dc3d42550b2bbd0d
Normal-8382.png,Normal,Normal/images/Normal-8382.png,Normal/masks/Normal-8382.png,True,G17808,train,2e6fb069124a2e06fa9e4bf1f39e5098,8969c71a19453bf9
Normal-8383.png,Normal,Normal/images/Normal-8383.png,Normal/masks/Normal-8383.png,True,G17809,test,fd6490e41a3c60926d3d446b82642a13,e81f974b1c494b17
Normal-8384.png,Normal,Normal/images/Normal-8384.png,Normal/masks/Normal-8384.png,True,G17810,test,90b2e321f4a96a2662fd97400e1a6564,896e061a5fec78e1
Normal-8385.png,Normal,Normal/images/Normal-8385.png,Normal/masks/Normal-8385.png,True,G17811,train,a366b7c46f8611f34397d8355adbf37c,917c871a1f453c4f
Normal-8386.png,Normal,Normal/images/Normal-8386.png,Normal/masks/Normal-8386.png,True,G17812,test,33dc653f4f669269dc6c8e75e18ff721,996885171f5f1e1a
Normal-8387.png,Normal,Normal/images/Normal-8387.png,Normal/masks/Normal-8387.png,True,G17813,train,f72d3dfaef6b8b4e2a0a5a21027be315,c83d420719d74f2f
Normal-8388.png,Normal,Normal/images/Normal-8388.png,Normal/masks/Normal-8388.png,True,G17814,train,90a2e5056bc32cddae6eb9b8947f316d,9979d61e150e3a1b
Normal-8389.png,Normal,Normal/images/Normal-8389.png,Normal/masks/Normal-8389.png,True,G17815,test,bddaca7a9b91ef856a16dc1a6c7e8537,80691e1a7b7f7a60
Normal-839.png,Normal,Normal/images/Normal-839.png,Normal/masks/Normal-839.png,True,G17816,train,725ed73d9b2e949ff665ca468b1ad725,d12dc21bd9693c69
Normal-8390.png,Normal,Normal/images/Normal-8390.png,Normal/masks/Normal-8390.png,True,G17817,test,d48833224c26cefc76f1ab3dd1b564c2,dd2c52d34b0f2f0a
Normal-8391.png,Normal,Normal/images/Normal-8391.png,Normal/masks/Normal-8391.png,True,G17818,train,86e598b669ba04ab9124aad91766881c,983b85168a4dbee9
Normal-8392.png,Normal,Normal/images/Normal-8392.png,Normal/masks/Normal-8392.png,True,G17819,train,874fa017cdd0cc79921dbb979bf498ed,dd3c570b1d0c19bc
Normal-8393.png,Normal,Normal/images/Normal-8393.png,Normal/masks/Normal-8393.png,True,G17820,train,c06a4a2a214e442c95268d4dedb9ba3d,dd3dc21b154a1b59
Normal-8394.png,Normal,Normal/images/Normal-8394.png,Normal/masks/Normal-8394.png,True,G17821,train,c32efd2a9a9483d6aa77cd0e8dd90604,c83d86523dedab11
Normal-8395.png,Normal,Normal/images/Normal-8395.png,Normal/masks/Normal-8395.png,True,G17822,train,e13c5dc85b3c27114fe0edaa0427ef8e,9969861617d71a97
Normal-8396.png,Normal,Normal/images/Normal-8396.png,Normal/masks/Normal-8396.png,True,G17823,train,72f635f3abdc3d9a4aac0c59abd6ada4,997bc71e194d3a11
Normal-8397.png,Normal,Normal/images/Normal-8397.png,Normal/masks/Normal-8397.png,True,G17824,train,73ecc4c54ed5ee4593b396e617d24e8a,887dc61e105f1f47
Normal-8398.png,Normal,Normal/images/Normal-8398.png,Normal/masks/Normal-8398.png,True,G17825,train,d39213cf0fee0d01c3e3f3778a8fc67f,d9f8c65a1ac65a98
Normal-8399.png,Normal,Normal/images/Normal-8399.png,Normal/masks/Normal-8399.png,True,G17826,val,5d399046f779b4c4fcc22cb91649988a,91f90e86199e6f98
Normal-84.png,Normal,Normal/images/Normal-84.png,Normal/masks/Normal-84.png,True,G17827,val,240a87c362e4a850a328e4fa730d86de,91758f1e8f0d0dcc
Normal-840.png,Normal,Normal/images/Normal-840.png,Normal/masks/Normal-840.png,True,G17828,val,d5ca83d33913d957c9aeccb627f4c55d,9d7d831a0b3c6c6c
Normal-8400.png,Normal,Normal/images/Normal-8400.png,Normal/masks/Normal-8400.png,True,G17829,test,e3502fd3cef3a7403831dc72d24165de,dd28934b5f8a495c
Normal-8401.png,Normal,Normal/images/Normal-8401.png,Normal/masks/Normal-8401.png,True,G17830,train,3d30ffa78d87bc6f315535c95f3b1b9a,9969071a5c5d7ae1
Normal-8402.png,Normal,Normal/images/Normal-8402.png,Normal/masks/Normal-8402.png,True,G17831,train,f7bd042cd0abd22542f6b4d824e5f491,c039571f1e6839f8
Normal-8403.png,Normal,Normal/images/Normal-8403.png,Normal/masks/Normal-8403.png,True,G17832,train,bd7480c1aeaebc0d6477b9e4ed27f800,8d2ec61b4c4e1d4f
Normal-8404.png,Normal,Normal/images/Normal-8404.png,Normal/masks/Normal-8404.png,True,G17833,train,5bcbedc0b52c34f6ff74f89357db4059,d92f061a194c79f9
Normal-8405.png,Normal,Normal/images/Normal-8405.png,Normal/masks/Normal-8405.png,True,G17834,train,fee8f592a672d0f95550051ce249065f,f97986161a0f171e
Normal-8406.png,Normal,Normal/images/Normal-8406.png,Normal/masks/Normal-8406.png,True,G17835,train,79f32e5b18ce2d0a085074d4ddf3a0a9,983f07184f5a795a
Normal-8407.png,Normal,Normal/images/Normal-8407.png,Normal/masks/Normal-8407.png,True,G17836,train,074966399b51e419ae5ad56e41a56e41,993bc61a1a4c5b79
Normal-8408.png,Normal,Normal/images/Normal-8408.png,Normal/masks/Normal-8408.png,True,G17837,train,9b4028af0f70143822398044ae87b0b3,d979c61a1c4a3b3c
Normal-8409.png,Normal,Normal/images/Normal-8409.png,Normal/masks/Normal-8409.png,True,G17838,train,57133c8ff18c1fc142732bec8a41892a,847e434b6cea6cac
Normal-841.png,Normal,Normal/images/Normal-841.png,Normal/masks/Normal-841.png,True,G17839,train,05d0f670d650283614f7270ca90e11d5,91a1c61ebce93d70
Normal-8410.png,Normal,Normal/images/Normal-8410.png,Normal/masks/Normal-8410.png,True,G17840,train,a490e64abffdd29620919ffefdfda3e3,d83f44171a1f4b0f
Normal-8411.png,Normal,Normal/images/Normal-8411.png,Normal/masks/Normal-8411.png,True,G17841,train,b945cdd2fb52f8001e9202d7c646c1b3,9979861a1d5f1a59
Normal-8412.png,Normal,Normal/images/Normal-8412.png,Normal/masks/Normal-8412.png,True,G17842,test,56e8469e46eba05ae4efa947c3352876,9979861a8d5938ed
Normal-8413.png,Normal,Normal/images/Normal-8413.png,Normal/masks/Normal-8413.png,True,G17843,train,65eb9c289ca4af4d30ae5f6086ec08e7,816904561a5f5f5f
Normal-8414.png,Normal,Normal/images/Normal-8414.png,Normal/masks/Normal-8414.png,True,G17844,train,6527ca1d24367a198fdfa5df97711bd6,816806161a5f5f7f
Normal-8415.png,Normal,Normal/images/Normal-8415.png,Normal/masks/Normal-8415.png,True,G17845,val,ae1b4a8bc275f57d085a34aa2f33b2f4,dd39c20f1d481f5c
Normal-8416.png,Normal,Normal/images/Normal-8416.png,Normal/masks/Normal-8416.png,True,G17846,train,d581bfa6d470882961a9d792398d6073,992806173ad77ad9
Normal-8417.png,Normal,Normal/images/Normal-8417.png,Normal/masks/Normal-8417.png,True,G17847,val,e497cf24d3af9481db1b9d879f37d435,d94d85161e4d3abc
Normal-8418.png,Normal,Normal/images/Normal-8418.png,Normal/masks/Normal-8418.png,True,G17848,train,e075b0e9f2ab6d488c896bdd2fa1d551,8179045e1f0f5f1b
Normal-8419.png,Normal,Normal/images/Normal-8419.png,Normal/masks/Normal-8419.png,True,G17849,train,e2d897881448c5008b38524995edc48e,c97b061a1f4d3a55
Normal-842.png,Normal,Normal/images/Normal-842.png,Normal/masks/Normal-842.png,True,G17850,val,d89ca5c49a326295cb8fcdedf29cbefc,9160867a967a397e
Normal-8420.png,Normal,Normal/images/Normal-8420.png,Normal/masks/Normal-8420.png,True,G17851,test,cc120d91bcb4e15c0e24cca7a944cfc3,9978471a194f39b9
Normal-8421.png,Normal,Normal/images/Normal-8421.png,Normal/masks/Normal-8421.png,True,G17852,train,acf5954152a484404425e6fcbf4f65d5,dd2ed0535b450a1f
Normal-8422.png,Normal,Normal/images/Normal-8422.png,Normal/masks/Normal-8422.png,True,G17853,train,3b2b6e11cf9005c992446fd114a0195a,99788792174d1a5f
Normal-8423.png,Normal,Normal/images/Normal-8423.png,Normal/masks/Normal-8423.png,True,G17854,train,df3adc389850775e396129b05b9c7551,d179c646911ec64f
Normal-8424.png,Normal,Normal/images/Normal-8424.png,Normal/masks/Normal-8424.png,True,G17855,train,510b250834ffc6fa3746cded9329b0f3,d92bc21b5d055b1d
Normal-8425.png,Normal,Normal/images/Normal-8425.png,Normal/masks/Normal-8425.png,True,G17856,train,0280708c46ce047747337772c6f611e0,9979861a94471bbd
Normal-8426.png,Normal,Normal/images/Normal-8426.png,Normal/masks/Normal-8426.png,True,G17857,train,e500ac0fd1ce1942d8f8776a28450fa9,8d2c4e4ee9cced44
Normal-8427.png,Normal,Normal/images/Normal-8427.png,Normal/masks/Normal-8427.png,True,G17858,train,4a0070d0f00c861819aa8bd6a94a28ea,9c6dc31a496e3ee0
Normal-8428.png,Normal,Normal/images/Normal-8428.png,Normal/masks/Normal-8428.png,True,G17859,train,3dca034c6ee9b232e70ea9185dcdb72d,987b87161e155e1b
Normal-8429.png,Normal,Normal/images/Normal-8429.png,Normal/masks/Normal-8429.png,True,G17860,train,a2ea820cdaeec3067ed690fdb2b10e08,9969071ac51e3aaf
Normal-843.png,Normal,Normal/images/Normal-843.png,Normal/masks/Normal-843.png,True,G17861,train,1a9345c6deec5762763d3852d7c7be39,d47cc31b8d0d3cc6
Normal-8430.png,Normal,Normal/images/Normal-8430.png,Normal/masks/Normal-8430.png,True,G17862,train,22edfe7d15ff6a6df3bdeb5c930972a8,d42f434b4c5d4a5d
Normal-8431.png,Normal,Normal/images/Normal-8431.png,Normal/masks/Normal-8431.png,True,G17863,val,b2e22ff6a69cba9029059cf56974f800,dd29d4821f455e17
Normal-8432.png,Normal,Normal/images/Normal-8432.png,Normal/masks/Normal-8432.png,True,G17864,test,d8e29bb7c2055e9e5d070258764445c2,993f061a196f68b9
Normal-8433.png,Normal,Normal/images/Normal-8433.png,Normal/masks/Normal-8433.png,True,G17865,val,4b90c7804a777a005232390289ae8ca7,887d075e1a496d7a
Normal-8434.png,Normal,Normal/images/Normal-8434.png,Normal/masks/Normal-8434.png,True,G17866,val,a06b46dc54faba56b470d0fc6668a70b,987f16184b1a3b3b
Normal-8435.png,Normal,Normal/images/Normal-8435.png,Normal/masks/Normal-8435.png,True,G17867,test,531ab64c865676e9333fb1e56d4ec7f7,f93d865e144f184d
Normal-8436.png,Normal,Normal/images/Normal-8436.png,Normal/masks/Normal-8436.png,True,G17868,test,8de2e5251973e4c804cba9d99025de7f,987b561b1d093bb8
Normal-8437.png,Normal,Normal/images/Normal-8437.png,Normal/masks/Normal-8437.png,True,G17869,train,bdbd6daebf363bfc90e77314488a6eb9,9d6c064b784e7879
Normal-8438.png,Normal,Normal/images/Normal-8438.png,Normal/masks/Normal-8438.png,True,G17870,train,c511db2592e888df3d816410a60c68c9,897b061e59e939e8
Normal-8439.png,Normal,Normal/images/Normal-8439.png,Normal/masks/Normal-8439.png,True,G17871,val,e2167df5eca86416683eba0ebcdd85b7,897d861a19ae6ab9
Normal-844.png,Normal,Normal/images/Normal-844.png,Normal/masks/Normal-844.png,True,G17872,val,16f75b4e4d6497f84e816eda6049d884,91f8de7a491919c6
Normal-8440.png,Normal,Normal/images/Normal-8440.png,Normal/masks/Normal-8440.png,True,G17873,test,0c581d3123e932e6742ddf46835f41c9,d129c6165b0def1a
Normal-8441.png,Normal,Normal/images/Normal-8441.png,Normal/masks/Normal-8441.png,True,G17874,test,5556172adf0a007ce49eac5c6c12f3c9,d93d42071f4c1b3d
Normal-8442.png,Normal,Normal/images/Normal-8442.png,Normal/masks/Normal-8442.png,True,G17875,train,8cbd9c2b296204999dd97eb0cea0cd82,997987031e575e1a
Normal-8443.png,Normal,Normal/images/Normal-8443.png,Normal/masks/Normal-8443.png,True,G17876,train,fd469e09def9631204fc38f4f593b7b3,dd2c061b851f195f
Normal-8444.png,Normal,Normal/images/Normal-8444.png,Normal/masks/Normal-8444.png,True,G17877,train,941d7245b15c0e8357033ead1084d9f6,9179861e1b4d5e78
Normal-8445.png,Normal,Normal/images/Normal-8445.png,Normal/masks/Normal-8445.png,True,G17878,train,fba110e16f0c0a21abab76add69b7175,983d931e3c161f93
Normal-8446.png,Normal,Normal/images/Normal-8446.png,Normal/masks/Normal-8446.png,True,G17879,train,ed133c35cc0753c04c4cfbc0e599fd0e,996f861a1d4f1ab1
Normal-8447.png,Normal,Normal/images/Normal-8447.png,Normal/masks/Normal-8447.png,True,G17880,train,192b00327b90372d815fb97153b9735c,996ec6191d2eb8ac
Normal-8448.png,Normal,Normal/images/Normal-8448.png,Normal/masks/Normal-8448.png,True,G17881,train,cffa553ac3051d80ba3caca5d9c95ae7,d12f46571b054f1d
Normal-8449.png,Normal,Normal/images/Normal-8449.png,Normal/masks/Normal-8449.png,True,G17882,train,fe3dd90a95c6115eb824297d96c9fe17,e90d86061f1f0f1f
Normal-845.png,Normal,Normal/images/Normal-845.png,Normal/masks/Normal-845.png,True,G17883,train,daf3e03842f70761657b6a6a8cf4f0e7,912bc61ad85979d3
Normal-8450.png,Normal,Normal/images/Normal-8450.png,Normal/masks/Normal-8450.png,True,G17884,train,0c9e2bf68ebdedf77863211cd5be6048,c03d0b52590eaf6f
Normal-8451.png,Normal,Normal/images/Normal-8451.png,Normal/masks/Normal-8451.png,True,G17885,val,ee620b0d3113f17232679849b805163a,d47f03086a3f2b33
Normal-8452.png,Normal,Normal/images/Normal-8452.png,Normal/masks/Normal-8452.png,True,G17886,train,11b914718d60358914c8e76fffd024a6,9928061a1f3b7b6b
Normal-8453.png,Normal,Normal/images/Normal-8453.png,Normal/masks/Normal-8453.png,True,G17887,train,bf757cacacb1f99a408a26b4cb1e4528,817b061e1a5b5f59
Normal-8454.png,Normal,Normal/images/Normal-8454.png,Normal/masks/Normal-8454.png,True,G17888,train,ecbce65d10cb2000b9c7742540dd4536,d0bfc257424b5f41
Normal-8455.png,Normal,Normal/images/Normal-8455.png,Normal/masks/Normal-8455.png,True,G17889,train,03368e8f6e81eb582f908cbebe14168d,89790e1a59e978e5
Normal-8456.png,Normal,Normal/images/Normal-8456.png,Normal/masks/Normal-8456.png,True,G17890,val,b7905fc8d7c6a03664651c8e7c57229d,d5fed253534a4b10
Normal-8457.png,Normal,Normal/images/Normal-8457.png,Normal/masks/Normal-8457.png,True,G17891,test,f92cd16d23b53e2b468ae090b3ee2971,99398716924f5f15
Normal-8458.png,Normal,Normal/images/Normal-8458.png,Normal/masks/Normal-8458.png,True,G17892,train,e83fe6ffa95b04cb278175008e1ebe95,c627c94f22ad6536
Normal-8459.png,Normal,Normal/images/Normal-8459.png,Normal/masks/Normal-8459.png,True,G17893,test,39ecc0c6990087845345d2d6eb568a3f,9128061b1fdc3fdc
Normal-846.png,Normal,Normal/images/Normal-846.png,Normal/masks/Normal-846.png,True,G17894,train,c05850bd735180d9d7ab26413808e844,956a8759cd6c68c6
Normal-8460.png,Normal,Normal/images/Normal-8460.png,Normal/masks/Normal-8460.png,True,G17895,train,c3b11ef7f24f9194ed838cda1667f827,c93dc24f1b296e38
Normal-8461.png,Normal,Normal/images/Normal-8461.png,Normal/masks/Normal-8461.png,True,G17896,train,e766588c21d713daf91d1b7b6341ffd5,817f574a1b493b38
Normal-8462.png,Normal,Normal/images/Normal-8462.png,Normal/masks/Normal-8462.png,True,G17897,train,3029808508234bee2f724d7d1c1c3744,996a161e1e1b1b3e
Normal-8463.png,Normal,Normal/images/Normal-8463.png,Normal/masks/Normal-8463.png,True,G17898,train,ca8141ba5e18a653dac561ee1a198ab8,9969c71e58471e59
Normal-8464.png,Normal,Normal/images/Normal-8464.png,Normal/masks/Normal-8464.png,True,G17899,train,02648a1fd3dfe29dfd929dd6dd80af55,dc2e424aec6c79e4
Normal-8465.png,Normal,Normal/images/Normal-8465.png,Normal/masks/Normal-8465.png,True,G17900,train,5693cb91e055c735c6a735ae29be3fcb,dd68d2c30b4dc89b
Normal-8466.png,Normal,Normal/images/Normal-8466.png,Normal/masks/Normal-8466.png,True,G17901,train,743e3e5f395690b8746b962500fe9a05,d969a5161f1f181e
Normal-8467.png,Normal,Normal/images/Normal-8467.png,Normal/masks/Normal-8467.png,True,G17902,train,65f6f7c2e23d748703c57ca341742264,b90d96185f5a38ea
Normal-8468.png,Normal,Normal/images/Normal-8468.png,Normal/masks/Normal-8468.png,True,G17903,train,49902898bd715575e51354b4effff1d5,c43c6643d71b9b4a
Normal-8469.png,Normal,Normal/images/Normal-8469.png,Normal/masks/Normal-8469.png,True,G17904,train,1cede3b57184ab999006a1838aabab49,cd2ed24b0d4c0fa7
Normal-847.png,Normal,Normal/images/Normal-847.png,Normal/masks/Normal-847.png,True,G17905,val,4de069d3872e27df879aec625ad2136b,812feb3acd4949c5
Normal-8470.png,Normal,Normal/images/Normal-8470.png,Normal/masks/Normal-8470.png,True,G17906,train,e59af554f76305a6b7f590ad4fc0c3c0,dd68875ac51ec81d
Normal-8471.png,Normal,Normal/images/Normal-8471.png,Normal/masks/Normal-8471.png,True,G17907,test,f93b63d33aa74ef36fb624b7ed311d4e,d43d420d1baaadb9
Normal-8472.png,Normal,Normal/images/Normal-8472.png,Normal/masks/Normal-8472.png,True,G17908,val,8e5ec562b9cc285ea9e460b8ebb5be0d,d829c5d71a0fc743
Normal-8473.png,Normal,Normal/images/Normal-8473.png,Normal/masks/Normal-8473.png,True,G17909,test,9e3afa50a808da89731b2c9351a22e89,d43c42db4b1b4b0f
Normal-8474.png,Normal,Normal/images/Normal-8474.png,Normal/masks/Normal-8474.png,True,G17910,train,34432b282daf34b1dfccb33a6b4769dc,d96cc69a41977a85
Normal-8475.png,Normal,Normal/images/Normal-8475.png,Normal/masks/Normal-8475.png,True,G17911,train,83a088999831bce2d20df9d68f841561,9968061e1a5f795d
Normal-8476.png,Normal,Normal/images/Normal-8476.png,Normal/masks/Normal-8476.png,True,G17912,train,8a80b342ee6f5514c269e9ba17de1e88,d43d425b1d463d39
Normal-8477.png,Normal,Normal/images/Normal-8477.png,Normal/masks/Normal-8477.png,True,G17913,train,5a684fedd8485144a5bc8732e283ff5e,d97b05461e0d4f1b
Normal-8478.png,Normal,Normal/images/Normal-8478.png,Normal/masks/Normal-8478.png,True,G17914,train,e82ae56148faf0e3c4957efd29a52b88,8920961d975e1d5f
Normal-8479.png,Normal,Normal/images/Normal-8479.png,Normal/masks/Normal-8479.png,True,G17915,train,1accac36195c13f8f1235b476b84bc8e,dc2dd24b0f4a0b3d
Normal-848.png,Normal,Normal/images/Normal-848.png,Normal/masks/Normal-848.png,True,G17916,test,ed18aac580b9a8c931d919d6ad7fc238,902feb3acd4c39c4
Normal-8480.png,Normal,Normal/images/Normal-8480.png,Normal/masks/Normal-8480.png,True,G17917,train,b54f0782a60afe0ad063443e81916d49,c13d464f184f6d39
Normal-8481.png,Normal,Normal/images/Normal-8481.png,Normal/masks/Normal-8481.png,True,G17918,train,f06770784d188f25f3dabf115b4d1dad,c03caf4b9c2cd4b3
Normal-8482.png,Normal,Normal/images/Normal-8482.png,Normal/masks/Normal-8482.png,True,G17919,train,a6d9e1b085792da21df509bd666f50cc,997a061a1f4f1a5d
Normal-8483.png,Normal,Normal/images/Normal-8483.png,Normal/masks/Normal-8483.png,True,G17920,train,986d567106b4271a9e96f3d5fe5bca1a,d969c61f944e1ad1
Normal-8484.png,Normal,Normal/images/Normal-8484.png,Normal/masks/Normal-8484.png,True,G17921,train,b782d204bd18b164123c912b5f2399f5,d92b44961b174f1e
Normal-8485.png,Normal,Normal/images/Normal-8485.png,Normal/masks/Normal-8485.png,True,G17922,train,9c006f4b816de8974303a54e0a273884,d93f85021e4e5a79
Normal-8486.png,Normal,Normal/images/Normal-8486.png,Normal/masks/Normal-8486.png,True,G17923,train,e2148741e7ee259de3de428814161f27,d93d421b0a4b1fbc
Normal-8487.png,Normal,Normal/images/Normal-8487.png,Normal/masks/Normal-8487.png,True,G17924,val,f0bca1c0ba36ca780b15e848062c2abe,dd2e82131b4d1f4b
Normal-8488.png,Normal,Normal/images/Normal-8488.png,Normal/masks/Normal-8488.png,True,G17925,val,cdbf3a32d46ee942a0c5dea72faad58e,d97b5616585b1d18
Normal-8489.png,Normal,Normal/images/Normal-8489.png,Normal/masks/Normal-8489.png,True,G17926,train,cec500cbaffe64723b3833129173f623,dc3fd35f400d0c0f
Normal-849.png,Normal,Normal/images/Normal-849.png,Normal/masks/Normal-849.png,True,G17927,test,d670ad3c9e1f3e463d8022f983427648,906de71ecd0d6c45
Normal-8490.png,Normal,Normal/images/Normal-8490.png,Normal/masks/Normal-8490.png,True,G17928,test,9ebef9fcc0a84e8d5956bf6a951d238e,9979841f1ec51b39
Normal-8491.png,Normal,Normal/images/Normal-8491.png,Normal/masks/Normal-8491.png,True,G17929,test,bfb0300190c5330ce853444de36ba24a,817a16111f5f781f
Normal-8492.png,Normal,Normal/images/Normal-8492.png,Normal/masks/Normal-8492.png,True,G17930,train,3a98ff6a3f9c4321a69795e6b8e6ead4,d02fc65b1d4c393c
Normal-8493.png,Normal,Normal/images/Normal-8493.png,Normal/masks/Normal-8493.png,True,G17931,train,d4f24684bc12c1c19aa11d8c5bd9e5c4,fc6cc31b1d445d1c
Normal-8494.png,Normal,Normal/images/Normal-8494.png,Normal/masks/Normal-8494.png,True,G17932,val,c1b971c8bc0e515f0c3362d798d492ba,cc7cc30b450a3ded
Normal-8495.png,Normal,Normal/images/Normal-8495.png,Normal/masks/Normal-8495.png,True,G17933,train,38d8658be119dcb37fa5c1717b90a680,d97d46c09a93c11f
Normal-8496.png,Normal,Normal/images/Normal-8496.png,Normal/masks/Normal-8496.png,True,G17934,val,3f724dcfeb1f39fe07e5f5e4e5f76031,d941961889bfe6b2
Normal-8497.png,Normal,Normal/images/Normal-8497.png,Normal/masks/Normal-8497.png,True,G17935,train,90ccb8d0c80fc4de4182dec939af507c,cd2dd00b4f0a3bad
Normal-8498.png,Normal,Normal/images/Normal-8498.png,Normal/masks/Normal-8498.png,True,G17936,train,01a7f3edd7419f12213be0faf4e4e353,997d861a196d3ae1
Normal-8499.png,Normal,Normal/images/Normal-8499.png,Normal/masks/Normal-8499.png,True,G17937,train,7b6dd94560f717641b795527db44d9f8,9178069718da7bd9
Normal-85.png,Normal,Normal/images/Normal-85.png,Normal/masks/Normal-85.png,True,G17938,train,4b7ee83d0118c410bd69f1f01b990862,d06ea73b8b0f2985
Normal-850.png,Normal,Normal/images/Normal-850.png,Normal/masks/Normal-850.png,True,G17939,train,099b69b5a1c15b72257377528a77ecd6,943fc61a9cc43e96
Normal-8500.png,Normal,Normal/images/Normal-8500.png,Normal/masks/Normal-8500.png,True,G17940,val,5448c195153fc95f43699bf8fee319e1,dc3f4255422f386b
Normal-8501.png,Normal,Normal/images/Normal-8501.png,Normal/masks/Normal-8501.png,True,G17941,val,4b285463bad89a0951d38ca86bade826,9179061e5ace7c1b
Normal-8502.png,Normal,Normal/images/Normal-8502.png,Normal/masks/Normal-8502.png,True,G17942,train,6962e2d27ca3dd86dffabc051ffb24ef,994fc65b492a0d2f
Normal-8503.png,Normal,Normal/images/Normal-8503.png,Normal/masks/Normal-8503.png,True,G17943,train,4080fc412f7eeb53d2304d0cfd4a3d6e,d42e42cf0cecee94
Normal-8504.png,Normal,Normal/images/Normal-8504.png,Normal/masks/Normal-8504.png,True,G17944,train,0b750a1e1fec3d3b7a72fe7f38a189ee,dd08c60f1d68bdb1
Normal-8505.png,Normal,Normal/images/Normal-8505.png,Normal/masks/Normal-8505.png,True,G17945,train,c9bc16465bff7c16ca5b475d895e8617,ac6cc31f551e1d45
Normal-8506.png,Normal,Normal/images/Normal-8506.png,Normal/masks/Normal-8506.png,True,G17946,train,8678f7dc2b166f66b951ef97428c1883,d83f4219ec39b0e3
Normal-8507.png,Normal,Normal/images/Normal-8507.png,Normal/masks/Normal-8507.png,True,G17947,train,2dd106352a669ff940a77450664d23d7,dd2ec20b1f443b1d
Normal-8508.png,Normal,Normal/images/Normal-8508.png,Normal/masks/Normal-8508.png,True,G17948,train,1e0c994cd4de6ec07ed8fa3cdaa1f8dd,9138c6d61b475f1c
Normal-8509.png,Normal,Normal/images/Normal-8509.png,Normal/masks/Normal-8509.png,True,G17949,test,1391581f933d06f1450bc0ba362c323d,811f473eff382886
Normal-851.png,Normal,Normal/images/Normal-851.png,Normal/masks/Normal-851.png,True,G17950,val,b3a8b446cbc947577fbd9ea9d2f2f650,91a9e61ada4b3c78
Normal-8510.png,Normal,Normal/images/Normal-8510.png,Normal/masks/Normal-8510.png,True,G17951,test,7a1aaad12e7357721f78c555a4b246dd,d42dc34b1e486f3c
Normal-8511.png,Normal,Normal/images/Normal-8511.png,Normal/masks/Normal-8511.png,True,G17952,test,c98a28010e8f4d86ad2aa1bc0e37d7cf,996dc61b184e39e9
Normal-8512.png,Normal,Normal/images/Normal-8512.png,Normal/masks/Normal-8512.png,True,G17953,train,b444cb7aa0f35ed26f21e1e1ea6ca5c1,9d2d861215cf3e8b
Normal-8513.png,Normal,Normal/images/Normal-8513.png,Normal/masks/Normal-8513.png,True,G17954,train,41d59dd06e6cf3d31abcf2b6e44fa682,d42e430b6c662eed
Normal-8514.png,Normal,Normal/images/Normal-8514.png,Normal/masks/Normal-8514.png,True,G17955,train,16c88216906dd2a9e81c419434c799ea,d96ec61b454a1d4d
Normal-8515.png,Normal,Normal/images/Normal-8515.png,Normal/masks/Normal-8515.png,True,G17956,train,db78278798a509bd3f63d8fb573f7a83,d93f4215d607194f
Normal-8516.png,Normal,Normal/images/Normal-8516.png,Normal/masks/Normal-8516.png,True,G17957,train,4ce2baac0615c5f08ab43ff310563cb2,d12f425f0a0e4b3f
Normal-8517.png,Normal,Normal/images/Normal-8517.png,Normal/masks/Normal-8517.png,True,G17958,train,95a81ce8a3b92bdcad8717e1cce92ec5,897d861a194f5e59
Normal-8518.png,Normal,Normal/images/Normal-8518.png,Normal/masks/Normal-8518.png,True,G17959,train,7fb5ed18f2c94a7514a7a6f3a324b563,882a87821b5fde5e
Normal-8519.png,Normal,Normal/images/Normal-8519.png,Normal/masks/Normal-8519.png,True,G17960,train,01bc747438d10c66e34ec12b773c86b2,992b061e5dcd78f0
Normal-852.png,Normal,Normal/images/Normal-852.png,Normal/masks/Normal-852.png,True,G17961,train,6ddb46c60c2a7b4e7bba25227a0abe58,943ebf3f85141097
Normal-8520.png,Normal,Normal/images/Normal-8520.png,Normal/masks/Normal-8520.png,True,G17962,train,16e9238e107aaa75e812a0ae0bdd6fa9,ddaa831d061e455f
Normal-8521.png,Normal,Normal/images/Normal-8521.png,Normal/masks/Normal-8521.png,True,G17963,train,b8c56821000cd6fda33d032046a99913,813b07161e597e7a
Normal-8522.png,Normal,Normal/images/Normal-8522.png,Normal/masks/Normal-8522.png,True,G17964,train,d144b1b0cb00e82ea2f84321f11b8ce9,d92d84131b455f1f
Normal-8523.png,Normal,Normal/images/Normal-8523.png,Normal/masks/Normal-8523.png,True,G17965,train,490ca4f6a0334a56d5c8c966122ec673,9921165f1f5e7831
Normal-8524.png,Normal,Normal/images/Normal-8524.png,Normal/masks/Normal-8524.png,True,G17966,val,23a0c51905bce4f452dc2c5d4597cd5f,9b7905161e155e5e
Normal-8525.png,Normal,Normal/images/Normal-8525.png,Normal/masks/Normal-8525.png,True,G17967,val,ba6257217e5213375f49da3ebe723a28,917f041e195f5b49
Normal-8526.png,Normal,Normal/images/Normal-8526.png,Normal/masks/Normal-8526.png,True,G17968,train,612311a6e94d792d9b03087188170a68,d42f4b4f4d4c2c5c
Normal-8527.png,Normal,Normal/images/Normal-8527.png,Normal/masks/Normal-8527.png,True,G17969,test,093c2ab62609ed6a767e212a8d37f164,dc2f42531f424f0f
Normal-8528.png,Normal,Normal/images/Normal-8528.png,Normal/masks/Normal-8528.png,True,G17970,train,dd13f54b903116640538ac89fee56fca,dd2c524b1f442d3d
Normal-8529.png,Normal,Normal/images/Normal-8529.png,Normal/masks/Normal-8529.png,True,G17971,train,78affb0a865a74aa22da08f711c1da56,d92cc61b455e39c5
Normal-853.png,Normal,Normal/images/Normal-853.png,Normal/masks/Normal-853.png,True,G17972,val,0a8104de709bf50408faef35de186138,817ade5a49793919
Normal-8530.png,Normal,Normal/images/Normal-8530.png,Normal/masks/Normal-8530.png,True,G17973,test,19c0868fd3cf1525865d2b1310be8a6b,807f56061a5f393b
Normal-8531.png,Normal,Normal/images/Normal-8531.png,Normal/masks/Normal-8531.png,True,G17974,test,3b0b64533ab6dd22c4094962b704a9b4,813946471a2eef3d
Normal-8532.png,Normal,Normal/images/Normal-8532.png,Normal/masks/Normal-8532.png,True,G17975,train,c7e9358648e56a1a76bae3194cf95055,d12a025f087e7dec
Normal-8533.png,Normal,Normal/images/Normal-8533.png,Normal/masks/Normal-8533.png,True,G17976,train,63709014fc8fa47d087afdd27c3439a9,d93946961e697938
Normal-8534.png,Normal,Normal/images/Normal-8534.png,Normal/masks/Normal-8534.png,True,G17977,train,a2625519435b9f9c3aac922fce3ccfb3,993f861b1b451f19
Normal-8535.png,Normal,Normal/images/Normal-8535.png,Normal/masks/Normal-8535.png,True,G17978,train,73e7bbf20cc990e9f5f97918b8e2218d,dd1a530f0c4bbd68
Normal-8536.png,Normal,Normal/images/Normal-8536.png,Normal/masks/Normal-8536.png,True,G17979,train,c1233097d291b42c025c57f7833a53ba,9979071a1d5a596d
Normal-8537.png,Normal,Normal/images/Normal-8537.png,Normal/masks/Normal-8537.png,True,G17980,train,7a7e72fce36a1fccf3f8c6da1fe65b28,993fc61b454a38cd
Normal-8538.png,Normal,Normal/images/Normal-8538.png,Normal/masks/Normal-8538.png,True,G17981,train,8ad44e636e13f5bd138debde0cb376ff,817b47161a4f5f1a
Normal-8539.png,Normal,Normal/images/Normal-8539.png,Normal/masks/Normal-8539.png,True,G17982,train,4aa410cad6e69545c61248cb768fb99b,c83dc61a9d4afd90
Normal-854.png,Normal,Normal/images/Normal-854.png,Normal/masks/Normal-854.png,True,G17983,train,6d2abbb99bbf2c62a8e2168a63a9ac4c,90abc61ade493dd1
Normal-8540.png,Normal,Normal/images/Normal-8540.png,Normal/masks/Normal-8540.png,True,G17984,train,dedcd292454464f2239486aabe6769d3,99689e1b5f78f082
Normal-8541.png,Normal,Normal/images/Normal-8541.png,Normal/masks/Normal-8541.png,True,G17985,train,840598e86e4f624dae3158882f4c7b4f,916b571e58297d29
Normal-8542.png,Normal,Normal/images/Normal-8542.png,Normal/masks/Normal-8542.png,True,G17986,train,da405431a37967ba8517a578f19aaf39,dd3d421b1c4e1bb8
Normal-8543.png,Normal,Normal/images/Normal-8543.png,Normal/masks/Normal-8543.png,True,G17987,val,ae2d5447837e07d76de8ef741d4d0f09,807f145a1b5b1bad
Normal-8544.png,Normal,Normal/images/Normal-8544.png,Normal/masks/Normal-8544.png,True,G17988,val,39a54bcedefb3761dba47adc4c21d56f,95688617bb4bc51e
Normal-8545.png,Normal,Normal/images/Normal-8545.png,Normal/masks/Normal-8545.png,True,G17989,test,6d8ca917703fa8beb53d80ca4fae32ba,dd3ac30f0c0f39b4
Normal-8546.png,Normal,Normal/images/Normal-8546.png,Normal/masks/Normal-8546.png,True,G17990,val,a07088435c1227bcb94f870b31b86a9f,897f861a0f1d5a4d
Normal-8547.png,Normal,Normal/images/Normal-8547.png,Normal/masks/Normal-8547.png,True,G17991,test,b92404b9b5b8c872268c9fac6bf8dbf9,dc6cc74b08ae3db0
Normal-8548.png,Normal,Normal/images/Normal-8548.png,Normal/masks/Normal-8548.png,True,G17992,train,d39a87302d17906b7664ada880689239,dd3dc6435f400f1a
Normal-8549.png,Normal,Normal/images/Normal-8549.png,Normal/masks/Normal-8549.png,True,G17993,test,97822b083a2b059dce533f4a7c06dbd1,dd3d440e1b4d6c78
Normal-855.png,Normal,Normal/images/Normal-855.png,Normal/masks/Normal-855.png,True,G17994,train,48b18400aa61216a63a7661e28f573da,c121c33bf86939e9
Normal-8550.png,Normal,Normal/images/Normal-8550.png,Normal/masks/Normal-8550.png,True,G17995,train,8f48074be8e5a3a9ff0484b8ca2b219e,9968871a81c77eb9
Normal-8551.png,Normal,Normal/images/Normal-8551.png,Normal/masks/Normal-8551.png,True,G17996,train,ddcd7f6a874ed674bb9d19d61ff3d823,8928c65c1f4f7a59
Normal-8552.png,Normal,Normal/images/Normal-8552.png,Normal/masks/Normal-8552.png,True,G17997,train,face8da00c208b7efdb70e223ffba147,997f46121b4d2f38
Normal-8553.png,Normal,Normal/images/Normal-8553.png,Normal/masks/Normal-8553.png,True,G17998,train,2c1d64f706768bc5047a97f791dd64fa,d82fc61b096e1f89
Normal-8554.png,Normal,Normal/images/Normal-8554.png,Normal/masks/Normal-8554.png,True,G17999,train,6829030cd9634fb6fccdc41cddee0a93,d97fc61a1d4a1919
Normal-8555.png,Normal,Normal/images/Normal-8555.png,Normal/masks/Normal-8555.png,True,G18000,train,ae74d6ca65d5a69972bf0b44b8949f56,c82fc64f1f94087d
Normal-8556.png,Normal,Normal/images/Normal-8556.png,Normal/masks/Normal-8556.png,True,G18001,train,d541dd481c4c5a6f7a1b67115741316f,dd2dd60b0e4a29bc
Normal-8557.png,Normal,Normal/images/Normal-8557.png,Normal/masks/Normal-8557.png,True,G18002,train,d65c286353444f01962f917f6c1ed0c1,dd2fd2431b0c4a1f
Normal-8558.png,Normal,Normal/images/Normal-8558.png,Normal/masks/Normal-8558.png,True,G18003,train,91036de7bda9650a3d37515693d61ecc,9928861a1bcf3bb9
Normal-8559.png,Normal,Normal/images/Normal-8559.png,Normal/masks/Normal-8559.png,True,G18004,train,55f455a794edcb206f3e4dff08e2acf0,9968d6071a8e4e5f
Normal-856.png,Normal,Normal/images/Normal-856.png,Normal/masks/Normal-856.png,True,G18005,train,60d1031aabfd7bb78c7c6df5f9c8a310,917c8318edfc19c9
Normal-8560.png,Normal,Normal/images/Normal-8560.png,Normal/masks/Normal-8560.png,True,G18006,train,0934b7785555279d68a73883e00ad30f,dc3d534f0e2c381d
Normal-8561.png,Normal,Normal/images/Normal-8561.png,Normal/masks/Normal-8561.png,True,G18007,val,1c6d9fb3633904f471d8ef4a40d7f825,917b05061f5d5a79
Normal-8562.png,Normal,Normal/images/Normal-8562.png,Normal/masks/Normal-8562.png,True,G18008,val,d5157a215111c25dcacbcd6b25d152bd,812b06155e5f796e
Normal-8563.png,Normal,Normal/images/Normal-8563.png,Normal/masks/Normal-8563.png,True,G18009,train,f47f3a07b6a5866fa25f7002d65ee154,853cd349181f6b3d
Normal-8564.png,Normal,Normal/images/Normal-8564.png,Normal/masks/Normal-8564.png,True,G18010,val,89b4c0451cf6058adbbf25361d1b1268,d43d0f0b1d4dcc4e
Normal-8565.png,Normal,Normal/images/Normal-8565.png,Normal/masks/Normal-8565.png,True,G18011,train,b6aa5c23e857011dde9f8dbc267e3dfb,f969c51315461e5e
Normal-8566.png,Normal,Normal/images/Normal-8566.png,Normal/masks/Normal-8566.png,True,G18012,test,66b8f2a0dcf123092967e45248d25390,d17d460a191f456f
Normal-8567.png,Normal,Normal/images/Normal-8567.png,Normal/masks/Normal-8567.png,True,G18013,train,a18c79c8c1119df01d65b225e7fbe55a,813b16185b6d78f9
Normal-8568.png,Normal,Normal/images/Normal-8568.png,Normal/masks/Normal-8568.png,True,G18014,train,7ef2ce2651d1854cdd2debb2bbc80a2f,883dc21b425e7bf4
Normal-8569.png,Normal,Normal/images/Normal-8569.png,Normal/masks/Normal-8569.png,True,G18015,train,2cf7deaa1cb2349afe3ca55bbbe9b61a,dc2cc34b0faca47c
Normal-857.png,Normal,Normal/images/Normal-857.png,Normal/masks/Normal-857.png,True,G18016,test,0bdb26ea048b19581eb224a85eba91d2,912fc71add4919c5
Normal-8570.png,Normal,Normal/images/Normal-8570.png,Normal/masks/Normal-8570.png,True,G18017,train,302ec0006393d9a9c64f5d2266bd0b1c,dc2ec25f504f1d45
Normal-8571.png,Normal,Normal/images/Normal-8571.png,Normal/masks/Normal-8571.png,True,G18018,test,2203c970ad7ab3fe28e049762443fa34,d93bd64e1e1a481e
Normal-8572.png,Normal,Normal/images/Normal-8572.png,Normal/masks/Normal-8572.png,True,G18019,train,a2be2a63411549a5e1692ef36a49ba4c,997f871a1d4f5a08
Normal-8573.png,Normal,Normal/images/Normal-8573.png,Normal/masks/Normal-8573.png,True,G18020,train,34b82ec4aae86ea7014837c1dd41f72a,d43c8343594f4f0f
Normal-8574.png,Normal,Normal/images/Normal-8574.png,Normal/masks/Normal-8574.png,True,G18021,train,4f87a61e60ef33ba226da90f598db0d3,d43d5315580e7d39
Normal-8575.png,Normal,Normal/images/Normal-8575.png,Normal/masks/Normal-8575.png,True,G18022,train,617097200cf52dd28f5d78fd4a9173a6,9168c61719df5cb8
Normal-8576.png,Normal,Normal/images/Normal-8576.png,Normal/masks/Normal-8576.png,True,G18023,train,6aeba2fa981f8284c7e1dc3abad6af67,d0799b13bc02b717
Normal-8577.png,Normal,Normal/images/Normal-8577.png,Normal/masks/Normal-8577.png,True,G18024,train,7576b1d16e442ec6a5e76378553d1d29,d139960f1e1d4c3d
Normal-8578.png,Normal,Normal/images/Normal-8578.png,Normal/masks/Normal-8578.png,True,G18025,train,ceab284354c0e8df67c098fce7383d02,817b561b792b390e
Normal-8579.png,Normal,Normal/images/Normal-8579.png,Normal/masks/Normal-8579.png,True,G18026,test,0eafa298d001090ec40d4b3895e7aa93,d43d42830b0fed3d
Normal-858.png,Normal,Normal/images/Normal-858.png,Normal/masks/Normal-858.png,True,G18027,train,379a69827180996cdb0224a839feeefd,806ad65a5c5137f6
Normal-8580.png,Normal,Normal/images/Normal-8580.png,Normal/masks/Normal-8580.png,True,G18028,val,bac8851be5f6b2acdfac8f3cdb8a9624,d13906161d5f8e1f
Normal-8581.png,Normal,Normal/images/Normal-8581.png,Normal/masks/Normal-8581.png,True,G18029,val,4eb22da254f631efacf13bd89d29ba57,993d061e145e5e5d
Normal-8582.png,Normal,Normal/images/Normal-8582.png,Normal/masks/Normal-8582.png,True,G18030,val,30c07271f04ce3aa80456b632c4d7fa3,897dc25a4a4f581f
Normal-8583.png,Normal,Normal/images/Normal-8583.png,Normal/masks/Normal-8583.png,True,G18031,test,7f2cd8c471b349c0fa49a56594b34af3,d92847161e5d4e5b
Normal-8584.png,Normal,Normal/images/Normal-8584.png,Normal/masks/Normal-8584.png,True,G18032,train,7d40fd0b17210137f48c3bfb00a817af,dc2f031b574f3868
Normal-8585.png,Normal,Normal/images/Normal-8585.png,Normal/masks/Normal-8585.png,True,G18033,test,ff5f5c5f5470ed65770456f9db617dcc,997b451616465e5e
Normal-8586.png,Normal,Normal/images/Normal-8586.png,Normal/masks/Normal-8586.png,True,G18034,train,4368b47f4d045d3d58e02dc7c2238fc9,99bf061a455a79c5
Normal-8587.png,Normal,Normal/images/Normal-8587.png,Normal/masks/Normal-8587.png,True,G18035,train,fad41ed0b1405534e968a83c2b768de8,9978871b130e3b8f
Normal-8588.png,Normal,Normal/images/Normal-8588.png,Normal/masks/Normal-8588.png,True,G18036,train,8ede186621920bc8a14a36e6ada57d3d,997986061f954f1b
Normal-8589.png,Normal,Normal/images/Normal-8589.png,Normal/masks/Normal-8589.png,True,G18037,train,fb412af503199901a645a3e8bbf74b77,d1ad469b079e3a85
Normal-859.png,Normal,Normal/images/Normal-859.png,Normal/masks/Normal-859.png,True,G18038,train,3082fa59eae7698d65cb32d3e957e023,997e461a5e1a584f
Normal-8590.png,Normal,Normal/images/Normal-8590.png,Normal/masks/Normal-8590.png,True,G18039,train,4fcd7413e316b0774cc6507ace010c51,d53d86431d0e463f
Normal-8591.png,Normal,Normal/images/Normal-8591.png,Normal/masks/Normal-8591.png,True,G18040,train,50dd7e91a25fc0def6e88a7cd34af264,996b061a5d1b79c5
Normal-8592.png,Normal,Normal/images/Normal-8592.png,Normal/masks/Normal-8592.png,True,G18041,train,765deabc64fe2c43d307dc24151022fd,897f061c1f184f3e
Normal-8593.png,Normal,Normal/images/Normal-8593.png,Normal/masks/Normal-8593.png,True,G18042,test,63ae48bb258605b72df178de645ba86a,913fc61a5d4c7a58
Normal-8594.png,Normal,Normal/images/Normal-8594.png,Normal/masks/Normal-8594.png,True,G18043,train,a1c9489f1b3ec42a33892cdee3fab99e,a3c22ce53eb4a0be
Normal-8595.png,Normal,Normal/images/Normal-8595.png,Normal/masks/Normal-8595.png,True,G18044,train,a9418ea10a81842d02bd33ddd086c5e9,dd2842d35bc32bd4
Normal-8596.png,Normal,Normal/images/Normal-8596.png,Normal/masks/Normal-8596.png,True,G18045,train,f88016aab03334c848cb8494aad4f886,dd2cc61a1d4c5f19
Normal-8597.png,Normal,Normal/images/Normal-8597.png,Normal/masks/Normal-8597.png,True,G18046,val,4ad8a0e4dc044c6aaec5e2c98c59a14d,d52d021c5a2d5b6f
Normal-8598.png,Normal,Normal/images/Normal-8598.png,Normal/masks/Normal-8598.png,True,G18047,train,542fa63a7ed94c2fee258848340c4a73,980d46125f4e1f5f
Normal-8599.png,Normal,Normal/images/Normal-8599.png,Normal/masks/Normal-8599.png,True,G18048,val,6d47efedc1ade14a2c812f37b9a98d79,9d2cc61b494578f9
Normal-86.png,Normal,Normal/images/Normal-86.png,Normal/masks/Normal-86.png,True,G18049,train,4b0378c21a5313d789d8890a663ae269,d03be61a784d19dc
Normal-860.png,Normal,Normal/images/Normal-860.png,Normal/masks/Normal-860.png,True,G18050,val,b3b649dfb818f78a5ea7d64248d618a1,812dce69384b3f5a
Normal-8600.png,Normal,Normal/images/Normal-8600.png,Normal/masks/Normal-8600.png,True,G18051,test,3df458350cea6ecb9c33f6eb9e98d0e3,806845561fc55f9f
Normal-8601.png,Normal,Normal/images/Normal-8601.png,Normal/masks/Normal-8601.png,True,G18052,train,89fa81bc896731c29cd2c1f5e15aff15,d42cc34b096d6f3c
Normal-8602.png,Normal,Normal/images/Normal-8602.png,Normal/masks/Normal-8602.png,True,G18053,train,f3f88327941ca2760e4e208feb20499a,842e53530b4f2fbc
Normal-8603.png,Normal,Normal/images/Normal-8603.png,Normal/masks/Normal-8603.png,True,G18054,test,f3b69cd747b2ba4557fc701b951e7c65,897d861a1cc51bbd
Normal-8604.png,Normal,Normal/images/Normal-8604.png,Normal/masks/Normal-8604.png,True,G18055,train,d59ee702cad1e33aafdc9170af5331aa,997b05961a11eb1f
Normal-8605.png,Normal,Normal/images/Normal-8605.png,Normal/masks/Normal-8605.png,True,G18056,val,43a63e49350a9d8394fe88eb41906b0a,897906531f5d6959
Normal-8606.png,Normal,Normal/images/Normal-8606.png,Normal/masks/Normal-8606.png,True,G18057,train,8d821d556261af43fd7e03913a19bbd6,d92516170f1b4f1e
Normal-8607.png,Normal,Normal/images/Normal-8607.png,Normal/masks/Normal-8607.png,True,G18058,train,7229f04e2cdbc03f301b453b89197c50,dc2ec35b0d472a0f
Normal-8608.png,Normal,Normal/images/Normal-8608.png,Normal/masks/Normal-8608.png,True,G18059,train,50f6e7fe0702bf78b9bad0df12d9ba66,997f871e58051e78
Normal-8609.png,Normal,Normal/images/Normal-8609.png,Normal/masks/Normal-8609.png,True,G18060,train,86acc80d4d23a96aad6bf60e86b93af9,dd3d42471b4e4d1c
Normal-861.png,Normal,Normal/images/Normal-861.png,Normal/masks/Normal-861.png,True,G18061,train,5fdc42c2332d426a80518d666ecec0a4,9139c65a9cc93db4
Normal-8610.png,Normal,Normal/images/Normal-8610.png,Normal/masks/Normal-8610.png,True,G18062,train,e611b340f3a0540ebc924bf38c7070de,997b861e1bca1a78
Normal-8611.png,Normal,Normal/images/Normal-8611.png,Normal/masks/Normal-8611.png,True,G18063,train,72ad5ce439e6049611e5a3f612172f35,817a1f06195f687b
Normal-8612.png,Normal,Normal/images/Normal-8612.png,Normal/masks/Normal-8612.png,True,G18064,train,e42b7447effa490f4b3b577453da5ca8,cc3d4a1f0d2cbcac
Normal-8613.png,Normal,Normal/images/Normal-8613.png,Normal/masks/Normal-8613.png,True,G18065,train,ff7715fe6cd92193b67f1e68f6ceaeef,de2dd1430d525d0f
Normal-8614.png,Normal,Normal/images/Normal-8614.png,Normal/masks/Normal-8614.png,True,G18066,train,cc31cfc7477bfbb490be26a422ff2a5a,c03d86431b5e4f1f
Normal-8615.png,Normal,Normal/images/Normal-8615.png,Normal/masks/Normal-8615.png,True,G18067,train,78eaf644266b1c1462a040ee29fc4ae3,b959421b092f38ef
Normal-8616.png,Normal,Normal/images/Normal-8616.png,Normal/masks/Normal-8616.png,True,G18068,val,a1d984ff73cb10dac99238e4984019ef,dc2cc61b454e5d4d
Normal-8617.png,Normal,Normal/images/Normal-8617.png,Normal/masks/Normal-8617.png,True,G18069,val,a1c6d19ef3c75518872b854273b7e472,d02c420f7b25fe39
Normal-8618.png,Normal,Normal/images/Normal-8618.png,Normal/masks/Normal-8618.png,True,G18070,test,5d24599000ce306ecd1a598d1e6868bb,c168065e1d4f5b3b
Normal-8619.png,Normal,Normal/images/Normal-8619.png,Normal/masks/Normal-8619.png,True,G18071,test,1cd8e71b37d51a5d92174ebbc3286ee4,9929061f4f5e18eb
Normal-862.png,Normal,Normal/images/Normal-862.png,Normal/masks/Normal-862.png,True,G18072,test,fc0de14ad7eaa7d56abc9518b1380341,907de63a296d3c4c
Normal-8620.png,Normal,Normal/images/Normal-8620.png,Normal/masks/Normal-8620.png,True,G18073,train,63f17088e69da1f9c686752908138c2a,897bc71e18c75e18
Normal-8621.png,Normal,Normal/images/Normal-8621.png,Normal/masks/Normal-8621.png,True,G18074,train,739c5d1a9c5ed8065ab1494dd286acd6,cc6d0683534d5d5e
Normal-8622.png,Normal,Normal/images/Normal-8622.png,Normal/masks/Normal-8622.png,True,G18075,train,2e007c9c247f7993431366552a00d423,d13d461e186b38f9
Normal-8623.png,Normal,Normal/images/Normal-8623.png,Normal/masks/Normal-8623.png,True,G18076,train,506aa1021d3420f838d3e5414d954f19,987f021a493d3abd
Normal-8624.png,Normal,Normal/images/Normal-8624.png,Normal/masks/Normal-8624.png,True,G18077,train,0a04862b7da7d9676b96153d3aaa1c9f,997f075a1f0a1b38
Normal-8625.png,Normal,Normal/images/Normal-8625.png,Normal/masks/Normal-8625.png,True,G18078,train,625d493c875d5dff527ab0b9d5ff5b2d,997a8711855e1b5e
Normal-8626.png,Normal,Normal/images/Normal-8626.png,Normal/masks/Normal-8626.png,True,G18079,train,eb35beaeba09f058a6b67f3c91e6e218,dd2dd21f1c403b39
Normal-8627.png,Normal,Normal/images/Normal-8627.png,Normal/masks/Normal-8627.png,True,G18080,train,8e37a7d5edb3b4df014b155395e2fac8,9939c61e1a4f2f38
Normal-8628.png,Normal,Normal/images/Normal-8628.png,Normal/masks/Normal-8628.png,True,G18081,train,cb7a5f8fecb6ad20e6c15e146f6b4bfa,91794616135fce5a
Normal-8629.png,Normal,Normal/images/Normal-8629.png,Normal/masks/Normal-8629.png,True,G18082,train,40a0bffba6b2abea17b85dded9045f5c,91609c1a5b1d797b
Normal-863.png,Normal,Normal/images/Normal-863.png,Normal/masks/Normal-863.png,True,G18083,train,0dea583869b8d1c3009a6a092f78a734,c13bee3a5c493949
Normal-8630.png,Normal,Normal/images/Normal-8630.png,Normal/masks/Normal-8630.png,True,G18084,train,4d6c4732dd7879755c2cef9194f484e9,817817871a9f5f89
Normal-8631.png,Normal,Normal/images/Normal-8631.png,Normal/masks/Normal-8631.png,True,G18085,train,666ad1d56b747996bb7863f012bdefaa,887bd71e944e1e1a
Normal-8632.png,Normal,Normal/images/Normal-8632.png,Normal/masks/Normal-8632.png,True,G18086,val,e521c821d8c68dce8347be4a845f4ad7,987b05161a5f5a5e
Normal-8633.png,Normal,Normal/images/Normal-8633.png,Normal/masks/Normal-8633.png,True,G18087,val,b1cdeed7a8f9f24aae269dc4653ea0c4,dd2e42510f0f2e3d
Normal-8634.png,Normal,Normal/images/Normal-8634.png,Normal/masks/Normal-8634.png,True,G18088,train,1aa5a505482a38fa95522d12cf67a329,dd2bd21f1e015f18
Normal-8635.png,Normal,Normal/images/Normal-8635.png,Normal/masks/Normal-8635.png,True,G18089,test,610242fe796095b2b1f7a3f5cb087c83,807fc71b1d4d1d58
Normal-8636.png,Normal,Normal/images/Normal-8636.png,Normal/masks/Normal-8636.png,True,G18090,train,e0902701b3073a66cbdbf12edd755594,d94c461b1b0e2b3f
Normal-8637.png,Normal,Normal/images/Normal-8637.png,Normal/masks/Normal-8637.png,True,G18091,train,be81457d4e1e13a7b65ed00e615ce5f4,81e85ed519cb399c
Normal-8638.png,Normal,Normal/images/Normal-8638.png,Normal/masks/Normal-8638.png,True,G18092,train,0c3c969991ee9251d63efd62c333ef71,d43d430b0a6f2b6b
Normal-8639.png,Normal,Normal/images/Normal-8639.png,Normal/masks/Normal-8639.png,True,G18093,val,207b90c047d32ac6c21b5a8b895d68a8,d13dc6425f0ec61b
Normal-864.png,Normal,Normal/images/Normal-864.png,Normal/masks/Normal-864.png,True,G18094,test,bcad8918f10acddf89496cbba07038b1,916dc71a8b6b38e4
Normal-8640.png,Normal,Normal/images/Normal-8640.png,Normal/masks/Normal-8640.png,True,G18095,test,d88a935c58d6237039e47dbe6698d43e,817f06465b1f4e4b
Normal-8641.png,Normal,Normal/images/Normal-8641.png,Normal/masks/Normal-8641.png,True,G18096,train,5e335be59176ffe26fce85f445719bc3,dd2dc6861f4dc81c
Normal-8642.png,Normal,Normal/images/Normal-8642.png,Normal/masks/Normal-8642.png,True,G18097,train,33cc67c7230a08f9a6e492c79ed01d22,dc3d430d0a2ebd93
Normal-8643.png,Normal,Normal/images/Normal-8643.png,Normal/masks/Normal-8643.png,True,G18098,test,f9a8b209db58f52723f41dc3a953965b,cd3ac317564f1e48
Normal-8644.png,Normal,Normal/images/Normal-8644.png,Normal/masks/Normal-8644.png,True,G18099,train,07f532e84d3deeee1f51cb2f46633fe5,cd58520f0d2f1dad
Normal-8645.png,Normal,Normal/images/Normal-8645.png,Normal/masks/Normal-8645.png,True,G18100,train,eebaa1144a3874ef5ee5246fb20e9331,c93d861a1d4f1a5d
Normal-8646.png,Normal,Normal/images/Normal-8646.png,Normal/masks/Normal-8646.png,True,G18101,train,f8503a4018521d5839a8bdae44ba22f2,996cc45b5d445b55
Normal-8647.png,Normal,Normal/images/Normal-8647.png,Normal/masks/Normal-8647.png,True,G18102,train,1ce327c290b5b07ebdd22edbd5b82efc,897b061a1d4d7b39
Normal-8648.png,Normal,Normal/images/Normal-8648.png,Normal/masks/Normal-8648.png,True,G18103,train,25c3d9eb9941dff6a7d10f11db7b3dc1,893b8718453ad9e7
Normal-8649.png,Normal,Normal/images/Normal-8649.png,Normal/masks/Normal-8649.png,True,G18104,train,c6af873d8345e873637700c7a7423bcb,dd2cd20b0f4a0dfc
Normal-865.png,Normal,Normal/images/Normal-865.png,Normal/masks/Normal-865.png,True,G18105,train,2518237e8b2dc4bee91e7c5fdf6d3def,913ed65a5d1919c5
Normal-8650.png,Normal,Normal/images/Normal-8650.png,Normal/masks/Normal-8650.png,True,G18106,train,de727eb939041069c6f8533a400379b7,d42e474f084c6e7d
Normal-8651.png,Normal,Normal/images/Normal-8651.png,Normal/masks/Normal-8651.png,True,G18107,train,e27db0391e62f178a8ebca26b92536bc,8179071a195f7a79
Normal-8652.png,Normal,Normal/images/Normal-8652.png,Normal/masks/Normal-8652.png,True,G18108,val,71645e2963b1ef3215cd9b3540eb2660,817a0e1a1b5b1f5e
Normal-8653.png,Normal,Normal/images/Normal-8653.png,Normal/masks/Normal-8653.png,True,G18109,val,7bb8ea460e3390bc590f723534bd8c49,917dc6d219c13b39
Normal-8654.png,Normal,Normal/images/Normal-8654.png,Normal/masks/Normal-8654.png,True,G18110,val,f6da1a541804bf623058290ed13e6353,987d061e075a5d5e
Normal-8655.png,Normal,Normal/images/Normal-8655.png,Normal/masks/Normal-8655.png,True,G18111,test,ecfbeb3e3ea5de77c7ce485497f213fd,d43d434b0d4af23d
Normal-8656.png,Normal,Normal/images/Normal-8656.png,Normal/masks/Normal-8656.png,True,G18112,test,1fc3091571ff698bf7abfb86d021af61,8979c746164d4f1e
Normal-8657.png,Normal,Normal/images/Normal-8657.png,Normal/masks/Normal-8657.png,True,G18113,train,66cd6a16717a22af5f508001217dc528,817f841a1d5a3b5b
Normal-8658.png,Normal,Normal/images/Normal-8658.png,Normal/masks/Normal-8658.png,True,G18114,train,d497aeef40cd22ee9961f2bd5ffb2bc3,d82f461f580d7a59
Normal-8659.png,Normal,Normal/images/Normal-8659.png,Normal/masks/Normal-8659.png,True,G18115,train,c90d13efbd2e8d8d64c3c3571b251395,992946171b1ede1b
Normal-866.png,Normal,Normal/images/Normal-866.png,Normal/masks/Normal-866.png,True,G18116,train,a4cba479eb2c431f0bd447db49eede7a,9d38a24a9f4b3dd0
Normal-8660.png,Normal,Normal/images/Normal-8660.png,Normal/masks/Normal-8660.png,True,G18117,train,c59aacc2570a7357dcb9e91ee10e5ce6,897887521fcf1a1b
Normal-8661.png,Normal,Normal/images/Normal-8661.png,Normal/masks/Normal-8661.png,True,G18118,train,50c020f92f134001fb8988e000126293,987b85161d1d5f1a
Normal-8662.png,Normal,Normal/images/Normal-8662.png,Normal/masks/Normal-8662.png,True,G18119,train,5dd52ea7e2dcb5c2973e1328421498ae,993b858690c51f9f
Normal-8663.png,Normal,Normal/images/Normal-8663.png,Normal/masks/Normal-8663.png,True,G18120,train,cd070dd6e0a8f429eb496ca32b50b422,dc2f425d4a6a38e5
Normal-8664.png,Normal,Normal/images/Normal-8664.png,Normal/masks/Normal-8664.png,True,G18121,train,7e935b8252b0a26730ba2f35099bc38e,dc2dc30f4c1f3d48
Normal-8665.png,Normal,Normal/images/Normal-8665.png,Normal/masks/Normal-8665.png,True,G18122,train,ff9075181e8129cec4aa0b010fc04977,c93dc6181a2f3da5
Normal-8666.png,Normal,Normal/images/Normal-8666.png,Normal/masks/Normal-8666.png,True,G18123,train,eefd034e9fef22bce9b22111c2a8fca4,807d064b5f4a5f5a
Normal-8667.png,Normal,Normal/images/Normal-8667.png,Normal/masks/Normal-8667.png,True,G18124,train,e5b5a045cd58a7425e9735d2a91d08d2,dd2a531d0d4d0d4f
Normal-8668.png,Normal,Normal/images/Normal-8668.png,Normal/masks/Normal-8668.png,True,G18125,val,9216e875043952fa81d6a91a17f66c6a,887f47161e4e7a38
Normal-8669.png,Normal,Normal/images/Normal-8669.png,Normal/masks/Normal-8669.png,True,G18126,train,7132996a340d7d67c60ccba717fdfa0a,ceb5c10faa2e8d15
Normal-867.png,Normal,Normal/images/Normal-867.png,Normal/masks/Normal-867.png,True,G18127,train,e7f441f87fc8f5c2c9ae99ac9183a9c8,d56f870b9f410bd0
Normal-8670.png,Normal,Normal/images/Normal-8670.png,Normal/masks/Normal-8670.png,True,G18128,test,268aeaaa436ff20574be70add60f9785,897d061e1f1d5a3a
Normal-8671.png,Normal,Normal/images/Normal-8671.png,Normal/masks/Normal-8671.png,True,G18129,test,d5b86444bc5c22c29bb9a6b9690b9fa5,813b161e59796b59
Normal-8672.png,Normal,Normal/images/Normal-8672.png,Normal/masks/Normal-8672.png,True,G18130,test,938d293da5426e11c3f02be5afa049ed,d528975a17561b55
Normal-8673.png,Normal,Normal/images/Normal-8673.png,Normal/masks/Normal-8673.png,True,G18131,train,15400c098178f35730a49e4288809f2d,816b0611574f1f5f
Normal-8674.png,Normal,Normal/images/Normal-8674.png,Normal/masks/Normal-8674.png,True,G18132,val,b9387df5acc26ac8289600f78ceaa8ed,813bd6161a1f5e1b
Normal-8675.png,Normal,Normal/images/Normal-8675.png,Normal/masks/Normal-8675.png,True,G18133,val,9faaf14d8a56bbbc3b2a26a609128411,997956041a5f6b39
Normal-8676.png,Normal,Normal/images/Normal-8676.png,Normal/masks/Normal-8676.png,True,G18134,train,4c32b29e8e790fdf511b1634f5ac6fad,997c461f441b5d4b
Normal-8677.png,Normal,Normal/images/Normal-8677.png,Normal/masks/Normal-8677.png,True,G18135,train,2de79351ce2360e94cdce07ff036c6ce,fd2cc61b95c33a11
Normal-8678.png,Normal,Normal/images/Normal-8678.png,Normal/masks/Normal-8678.png,True,G18136,train,ac5b4b19fc084b273a1e4cc291cdd0ac,894d861e5d6b788d
Normal-8679.png,Normal,Normal/images/Normal-8679.png,Normal/masks/Normal-8679.png,True,G18137,train,90f3e7f45c179ba678bbb0b0ec727bc0,9978461b190f3b3d
Normal-868.png,Normal,Normal/images/Normal-868.png,Normal/masks/Normal-868.png,True,G18138,val,55da08b5574174651c1983bc5437845f,d129c27ade4a5978
Normal-8680.png,Normal,Normal/images/Normal-8680.png,Normal/masks/Normal-8680.png,True,G18139,train,bfc578a18e90bb1afd70da06c0cc7174,dc3dc2db5cc00b99
Normal-8681.png,Normal,Normal/images/Normal-8681.png,Normal/masks/Normal-8681.png,True,G18140,train,4c61f0677290a48a15ab194234041ef3,81784e161b5d787b
Normal-8682.png,Normal,Normal/images/Normal-8682.png,Normal/masks/Normal-8682.png,True,G18141,val,b49787d3bb198db6ca0362d0a90126d2,817907965a957e99
Normal-8683.png,Normal,Normal/images/Normal-8683.png,Normal/masks/Normal-8683.png,True,G18142,train,eccd7c09e288b1ef103f766e778f2d52,d96bc71a5d487838
Normal-8684.png,Normal,Normal/images/Normal-8684.png,Normal/masks/Normal-8684.png,True,G18143,train,683ee66f6b9b1b34c4a24d60f6da4729,d929c7461e4c4b1f
Normal-8685.png,Normal,Normal/images/Normal-8685.png,Normal/masks/Normal-8685.png,True,G18144,train,da8fe1589a1c161e9e7246dccc78952b,dc3dc3431d57421b
Normal-8686.png,Normal,Normal/images/Normal-8686.png,Normal/masks/Normal-8686.png,True,G18145,train,c9b774c071175f96f7a7804ee70a154f,dc2d42530f2a2f3d
Normal-8687.png,Normal,Normal/images/Normal-8687.png,Normal/masks/Normal-8687.png,True,G18146,train,ec0ddd286be3b101208cd8e55d598301,d879079618893fb9
Normal-8688.png,Normal,Normal/images/Normal-8688.png,Normal/masks/Normal-8688.png,True,G18147,train,323128f3c99a93a7c867bdc1f5e809ed,883f025f1f421f3d
Normal-8689.png,Normal,Normal/images/Normal-8689.png,Normal/masks/Normal-8689.png,True,G18148,train,5273fc5198366ec9cb49b1ee517e8fcd,89798653154f5b1d
Normal-869.png,Normal,Normal/images/Normal-869.png,Normal/masks/Normal-869.png,True,G18149,train,ed1bd254524354ad300533ba6074891d,815f865ac71dc2a7
Normal-8690.png,Normal,Normal/images/Normal-8690.png,Normal/masks/Normal-8690.png,True,G18150,val,2d699f5995e92ab04a3c113bb94227ef,817884ce5f7b4b29
Normal-8691.png,Normal,Normal/images/Normal-8691.png,Normal/masks/Normal-8691.png,True,G18151,test,b039173dbc58c8f05442af97a56205d1,c43d15c7c31b5d1c
Normal-8692.png,Normal,Normal/images/Normal-8692.png,Normal/masks/Normal-8692.png,True,G18152,test,9e3150710c56315c3ce1e119ece232af,dd3cc253ddc2411d
Normal-8693.png,Normal,Normal/images/Normal-8693.png,Normal/masks/Normal-8693.png,True,G18153,test,cca79aba678449ca131e506ac126d186,99291618f9e9b0e7
Normal-8694.png,Normal,Normal/images/Normal-8694.png,Normal/masks/Normal-8694.png,True,G18154,train,6152785ae66018c55c720b992d22e72f,a94c961a494f7b6c
Normal-8695.png,Normal,Normal/images/Normal-8695.png,Normal/masks/Normal-8695.png,True,G18155,train,640f67edef46113260b744d4885abfcc,887bc7d65a444e5a
Normal-8696.png,Normal,Normal/images/Normal-8696.png,Normal/masks/Normal-8696.png,True,G18156,train,6697f2f311321ad7df01d04504d03e8f,916916161f5e7879
Normal-8697.png,Normal,Normal/images/Normal-8697.png,Normal/masks/Normal-8697.png,True,G18157,train,f534a399521f252f583b172c4decd0b4,990a161e5e7e3a78
Normal-8698.png,Normal,Normal/images/Normal-8698.png,Normal/masks/Normal-8698.png,True,G18158,train,b5da681ca6cea687592e93664b10de45,893f861a1c4e7979
Normal-8699.png,Normal,Normal/images/Normal-8699.png,Normal/masks/Normal-8699.png,True,G18159,train,4a1a2589e662fa92361f86d5ad45067d,d43d43574b0a2f0f
Normal-87.png,Normal,Normal/images/Normal-87.png,Normal/masks/Normal-87.png,True,G18160,train,65371a377f167d20fb04113c6cc970b0,9139c21eda5b19d9
Normal-870.png,Normal,Normal/images/Normal-870.png,Normal/masks/Normal-870.png,True,G18161,train,f7a3ea74e9ceacf588ec3f2ee3f13588,d43dc21fad2c3c64
Normal-8700.png,Normal,Normal/images/Normal-8700.png,Normal/masks/Normal-8700.png,True,G18162,train,ffd49bd656ca9273e973ec916ca975b2,c43c8b4bbc4bec1c
Normal-8701.png,Normal,Normal/images/Normal-8701.png,Normal/masks/Normal-8701.png,True,G18163,train,d462d9096765641c3bdd6937bde8da09,8a7d041e061fb8bf
Normal-8702.png,Normal,Normal/images/Normal-8702.png,Normal/masks/Normal-8702.png,True,G18164,train,c6f472113e0f72b61f44e5c955970a9b,d1298e58191f7a3b
Normal-8703.png,Normal,Normal/images/Normal-8703.png,Normal/masks/Normal-8703.png,True,G18165,train,07ee27ceea345bcc9ceb7cc7c9dde8a2,99680e1b4c78edec
Normal-8704.png,Normal,Normal/images/Normal-8704.png,Normal/masks/Normal-8704.png,True,G18166,train,b5f68855fada8640ac12f0f037eeac65,887985171e8e5f5a
Normal-8705.png,Normal,Normal/images/Normal-8705.png,Normal/masks/Normal-8705.png,True,G18167,val,119c52c2c144d9378b08f2ad71327c1d,897d061e5d6a7e28
Normal-8706.png,Normal,Normal/images/Normal-8706.png,Normal/masks/Normal-8706.png,True,G18168,val,17c73ea4cb0d595e8d579c9d41ab309d,dc59c30f0e0d2db5
Normal-8707.png,Normal,Normal/images/Normal-8707.png,Normal/masks/Normal-8707.png,True,G18169,val,22681609c5b680d3c31f625c491e2128,8979c78616c77e90
Normal-8708.png,Normal,Normal/images/Normal-8708.png,Normal/masks/Normal-8708.png,True,G18170,test,eb857707ac642e5ffce82e167fabb622,917d0f47190b0f1f
Normal-8709.png,Normal,Normal/images/Normal-8709.png,Normal/masks/Normal-8709.png,True,G18171,train,ebabba629140b12f7415989456899fcd,dd2cd21b450f584f
Normal-871.png,Normal,Normal/images/Normal-871.png,Normal/masks/Normal-871.png,True,G18172,test,4eb7b95cc9340d8f586c41da36b2b23f,896f471a0e4d3ce5
Normal-8710.png,Normal,Normal/images/Normal-8710.png,Normal/masks/Normal-8710.png,True,G18173,train,5a18bdd07e7809b58888926e9267dcee,887f071c1a1f5a3b
Normal-8711.png,Normal,Normal/images/Normal-8711.png,Normal/masks/Normal-8711.png,True,G18174,test,b47fa319d25db01dd16cb4ed29460f2f,997b841e1f4f5a18
Normal-8712.png,Normal,Normal/images/Normal-8712.png,Normal/masks/Normal-8712.png,True,G18175,val,7cc53c0e34d152a30e8760a2ad27b9db,903e16171e167cbe
Normal-8713.png,Normal,Normal/images/Normal-8713.png,Normal/masks/Normal-8713.png,True,G18176,train,fdd777efeb4f02a4e3b3381e3efb792c,d42e835f086e787a
Normal-8714.png,Normal,Normal/images/Normal-8714.png,Normal/masks/Normal-8714.png,True,G18177,train,1d895204c05d658e47cc997f60495123,9939e796c106b2e3
Normal-8715.png,Normal,Normal/images/Normal-8715.png,Normal/masks/Normal-8715.png,True,G18178,train,53a1dce018b363443df02918f146249e,d939d6561c051f39
Normal-8716.png,Normal,Normal/images/Normal-8716.png,Normal/masks/Normal-8716.png,True,G18179,val,bdb35b2b763c9169a75698b6b2b913bf,cd2cd2435b4d4e1d
Normal-8717.png,Normal,Normal/images/Normal-8717.png,Normal/masks/Normal-8717.png,True,G18180,train,3406e89a9d9441fa9af46e07c3b3c33f,d93dc21f840e3de8
Normal-8718.png,Normal,Normal/images/Normal-8718.png,Normal/masks/Normal-8718.png,True,G18181,train,6be2468f515f94916c66d510a1092062,8c6e471b5d4d285d
Normal-8719.png,Normal,Normal/images/Normal-8719.png,Normal/masks/Normal-8719.png,True,G18182,train,f0a38ee9cb33cb9af5f8b2b8e1b93e12,9c4cc21bc91fb8ad
Normal-872.png,Normal,Normal/images/Normal-872.png,Normal/masks/Normal-872.png,True,G18183,train,f6372f7235310dc58775c64d6f692548,c05fef0d0acd5a49
Normal-8720.png,Normal,Normal/images/Normal-8720.png,Normal/masks/Normal-8720.png,True,G18184,train,2bb3399268369879d1b58f694da00cdb,c82d8712975e1d4f
Normal-8721.png,Normal,Normal/images/Normal-8721.png,Normal/masks/Normal-8721.png,True,G18185,train,01c5e66a60567deabe4f5eb07a162130,997d861a181f5a5b
Normal-8722.png,Normal,Normal/images/Normal-8722.png,Normal/masks/Normal-8722.png,True,G18186,train,1657276cdcf42942cdb158bd6b7eda31,dd2d42531b425f1b
Normal-8723.png,Normal,Normal/images/Normal-8723.png,Normal/masks/Normal-8723.png,True,G18187,train,be7f5c0a100b40b79bd3a77b93aa7e14,9969c6161ac77f90
Normal-8724.png,Normal,Normal/images/Normal-8724.png,Normal/masks/Normal-8724.png,True,G18188,train,fb2ae8bb84616123bddf21c669fcf14c,d939d6061b05ed1b
Normal-8725.png,Normal,Normal/images/Normal-8725.png,Normal/masks/Normal-8725.png,True,G18189,val,cd44a9c606f5dda9cc181e4056a454c0,815f86185d5b1f4e
Normal-8726.png,Normal,Normal/images/Normal-8726.png,Normal/masks/Normal-8726.png,True,G18190,test,dbfaed5dbcae12fc12b3b8272bb85d99,d83dd2c39bc1c398
Normal-8727.png,Normal,Normal/images/Normal-8727.png,Normal/masks/Normal-8727.png,True,G18191,test,1875557f8dc9741413813631cffbf24f,8978171a59edb8a5
Normal-8728.png,Normal,Normal/images/Normal-8728.png,Normal/masks/Normal-8728.png,True,G18192,test,d477641893a2f8b6ce38db93228b1eaf,997d861e1c4b1f4a
Normal-8729.png,Normal,Normal/images/Normal-8729.png,Normal/masks/Normal-8729.png,True,G18193,train,dac5182e61bf8109fec91bf850e47e08,bd2d4243192ce4fb
Normal-873.png,Normal,Normal/images/Normal-873.png,Normal/masks/Normal-873.png,True,G18194,train,cc4f6a133a03c3c7241278172781fb29,dd28c20b9d693d74
Normal-8730.png,Normal,Normal/images/Normal-8730.png,Normal/masks/Normal-8730.png,True,G18195,train,3c8a40c99e486c7549e247b9fe5f5926,817fc61a155f384d
Normal-8731.png,Normal,Normal/images/Normal-8731.png,Normal/masks/Normal-8731.png,True,G18196,train,6afb61b59ded50e1dd3b3f1a77ffa3c6,dc3dc30f1f4a4d18
Normal-8732.png,Normal,Normal/images/Normal-8732.png,Normal/masks/Normal-8732.png,True,G18197,train,ee733cdeac75494dcfc677c040edba07,cc3dd40b1d4c1b7c
Normal-8733.png,Normal,Normal/images/Normal-8733.png,Normal/masks/Normal-8733.png,True,G18198,train,ce02a7a5699977eb801a297344086cb8,9c7dc61b154f184d
Normal-8734.png,Normal,Normal/images/Normal-8734.png,Normal/masks/Normal-8734.png,True,G18199,train,0a4a151e260afe6bd8480fdfd727a6b5,dc2cd30f8e0bbd82
Normal-8735.png,Normal,Normal/images/Normal-8735.png,Normal/masks/Normal-8735.png,True,G18200,train,8363a3355f93a878fd0eb5aaa23f8ecb,dc2cd243d3c7431b
Normal-8736.png,Normal,Normal/images/Normal-8736.png,Normal/masks/Normal-8736.png,True,G18201,train,50968a575fb534164d3d8035093c3792,91788618195e7bf9
Normal-8737.png,Normal,Normal/images/Normal-8737.png,Normal/masks/Normal-8737.png,True,G18202,train,446c2045172624181e9b73032da89d07,886d861e1f5a1d5e
Normal-8738.png,Normal,Normal/images/Normal-8738.png,Normal/masks/Normal-8738.png,True,G18203,train,a70cf1fbc41c6ecd54bc158ba874cd1f,dd2c420b4d7ebc64
Normal-8739.png,Normal,Normal/images/Normal-8739.png,Normal/masks/Normal-8739.png,True,G18204,train,b8c7f333d3ed2b7e8b03e7323cc4dab3,d93dc61b1d4b1859
Normal-874.png,Normal,Normal/images/Normal-874.png,Normal/masks/Normal-874.png,True,G18205,train,d96b73c6efbfd6014b89c37077097d8e,9178c61a9dcb39c6
Normal-8740.png,Normal,Normal/images/Normal-8740.png,Normal/masks/Normal-8740.png,True,G18206,train,bc3d9666ded76d93eb7ce0a6e5aa11b1,dd2e825fd40f0d43
Normal-8741.png,Normal,Normal/images/Normal-8741.png,Normal/masks/Normal-8741.png,True,G18207,val,351c6da162218a158269486024b64da0,986d86528f1b8f1d
Normal-8742.png,Normal,Normal/images/Normal-8742.png,Normal/masks/Normal-8742.png,True,G18208,val,ab944faecb66a4f35723abf65c4f2642,dd2a020b432f6d3f
Normal-8743.png,Normal,Normal/images/Normal-8743.png,Normal/masks/Normal-8743.png,True,G18209,val,2bb90db72a9c48232fd7fa9bf5d8668b,992986161f4e1f1f
Normal-8744.png,Normal,Normal/images/Normal-8744.png,Normal/masks/Normal-8744.png,True,G18210,test,daf306bd3d9ccb6dc9e34d69279ffd8c,c87a15c75b1e1ec2
Normal-8745.png,Normal,Normal/images/Normal-8745.png,Normal/masks/Normal-8745.png,True,G18211,train,15e3d60b8b2a4f1c165977d41331c690,dc3dc24b1d420f7c
Normal-8746.png,Normal,Normal/images/Normal-8746.png,Normal/masks/Normal-8746.png,True,G18212,train,0812e23e073a7e4d9f902bbe731cf4b8,b90c061b4d6c78ef
Normal-8747.png,Normal,Normal/images/Normal-8747.png,Normal/masks/Normal-8747.png,True,G18213,train,c06a841fa39d1469d7259a37009cc8ff,dd29421fc419b8eb
Normal-8748.png,Normal,Normal/images/Normal-8748.png,Normal/masks/Normal-8748.png,True,G18214,test,1780a97fdcb2694092d407be5f433f42,d93d4b1c110f2db9
Normal-8749.png,Normal,Normal/images/Normal-8749.png,Normal/masks/Normal-8749.png,True,G18215,train,e048a05183fe8359818651732c809750,917946865a1dad4f
Normal-875.png,Normal,Normal/images/Normal-875.png,Normal/masks/Normal-875.png,True,G18216,val,9f928c87562372245ac0ebab03edda7c,d528c20b9fe83df0
Normal-8750.png,Normal,Normal/images/Normal-8750.png,Normal/masks/Normal-8750.png,True,G18217,test,d874cd847d36254bf7daa8a0ac9044fb,d42e415b095e2e5f
Normal-8751.png,Normal,Normal/images/Normal-8751.png,Normal/masks/Normal-8751.png,True,G18218,train,9638d61491e02b9eb3d2d6de7aa2f700,dc2d42530fc82f9d
Normal-8752.png,Normal,Normal/images/Normal-8752.png,Normal/masks/Normal-8752.png,True,G18219,train,faf5e2f467ae8e4603259c487ff97d98,883dc2d34b1d4f0f
Normal-8753.png,Normal,Normal/images/Normal-8753.png,Normal/masks/Normal-8753.png,True,G18220,train,99fc5f5dafe8b5db1f8925d11334e26d,9979071e79a97838
Normal-8754.png,Normal,Normal/images/Normal-8754.png,Normal/masks/Normal-8754.png,True,G18221,train,13ee441ee0c34183a00b9af50db65b93,a97f16180d795d58
Normal-8755.png,Normal,Normal/images/Normal-8755.png,Normal/masks/Normal-8755.png,True,G18222,train,98479e1c0a0b72af5f6435e9691d1b38,dd29c5861f0d571a
Normal-8756.png,Normal,Normal/images/Normal-8756.png,Normal/masks/Normal-8756.png,True,G18223,train,73d880ed9a99d097b4f72b2419f014e3,8879575f1b0d2b38
Normal-8757.png,Normal,Normal/images/Normal-8757.png,Normal/masks/Normal-8757.png,True,G18224,train,247b6733c9658d9d6cb6a262ff4ad433,d57c420b1b4e391f
Normal-8758.png,Normal,Normal/images/Normal-8758.png,Normal/masks/Normal-8758.png,True,G18225,val,f55f7d0f42c012d98e4d92a998b45e6c,842ec71b4d5e2c67
Normal-8759.png,Normal,Normal/images/Normal-8759.png,Normal/masks/Normal-8759.png,True,G18226,test,9500adcb7ca3a641807fdc1cb97d9819,990c861a5f5e7a4e
Normal-876.png,Normal,Normal/images/Normal-876.png,Normal/masks/Normal-876.png,True,G18227,train,ad667c4a50103cfc9976baadb2352d54,916dc71aed692c45
Normal-8760.png,Normal,Normal/images/Normal-8760.png,Normal/masks/Normal-8760.png,True,G18228,train,d93109ab133f07675f3b73ce58efeeb0,817f8494121f3b3f
Normal-8761.png,Normal,Normal/images/Normal-8761.png,Normal/masks/Normal-8761.png,True,G18229,train,7cfb93e4e00786c8d6fb9967a83dd5f2,897f061e1c1f1b4d
Normal-8762.png,Normal,Normal/images/Normal-8762.png,Normal/masks/Normal-8762.png,True,G18230,val,19475df9bc495924a7ffe0b67f07c2ae,dd28d6421e4f4a1f
Normal-8763.png,Normal,Normal/images/Normal-8763.png,Normal/masks/Normal-8763.png,True,G18231,test,2b30d734ffc793748235d413b1214070,dd28d11717471e1d
Normal-8764.png,Normal,Normal/images/Normal-8764.png,Normal/masks/Normal-8764.png,True,G18232,test,ff9cd98fd4d31cc143c8fe6e0e096315,dd2a070c7c7d60f8
Normal-8765.png,Normal,Normal/images/Normal-8765.png,Normal/masks/Normal-8765.png,True,G18233,train,9c7e081197a132b4399bc675b1a6dc51,b90d861aedfcb0a1
Normal-8766.png,Normal,Normal/images/Normal-8766.png,Normal/masks/Normal-8766.png,True,G18234,train,5cb445c465a03e419ad091d206053c8f,dc2ec21bc14f1e4d
Normal-8767.png,Normal,Normal/images/Normal-8767.png,Normal/masks/Normal-8767.png,True,G18235,train,75604f608a94869e0c4feee13d066e3b,d03d43c30b9d0fb9
Normal-8768.png,Normal,Normal/images/Normal-8768.png,Normal/masks/Normal-8768.png,True,G18236,train,406eb23e18efa27ec70cd075691e967d,c939d65a1da8a3b4
Normal-8769.png,Normal,Normal/images/Normal-8769.png,Normal/masks/Normal-8769.png,True,G18237,train,ebc6d4da960cb27ba077b4cc3805144a,c97d865a455a584f
Normal-877.png,Normal,Normal/images/Normal-877.png,Normal/masks/Normal-877.png,True,G18238,train,678b5aa445da869f5af25cb077d613f5,d06f4f5d1c419e86
Normal-8770.png,Normal,Normal/images/Normal-8770.png,Normal/masks/Normal-8770.png,True,G18239,train,7503bea818013cec4b19ed84320d7256,fd28d79adb411055
Normal-8771.png,Normal,Normal/images/Normal-8771.png,Normal/masks/Normal-8771.png,True,G18240,train,a82c56ad2a553f9bad4894d7e093fd33,d46e035b1b4a2f3c
Normal-8772.png,Normal,Normal/images/Normal-8772.png,Normal/masks/Normal-8772.png,True,G18241,train,0533a93c914cd05617172801cbd6fd5e,d42ec70b086d6cee
Normal-8773.png,Normal,Normal/images/Normal-8773.png,Normal/masks/Normal-8773.png,True,G18242,train,3a3a8dc9a8c7fe54d9a779f6c57b6dac,897d86861e0f0f3d
Normal-8774.png,Normal,Normal/images/Normal-8774.png,Normal/masks/Normal-8774.png,True,G18243,train,c9251f4ce30a83c89a001bde21dc1871,997d8616184f5f1a
Normal-8775.png,Normal,Normal/images/Normal-8775.png,Normal/masks/Normal-8775.png,True,G18244,train,94781db320fe44006456554d67855cde,dc3d82071fc44f1b
Normal-8776.png,Normal,Normal/images/Normal-8776.png,Normal/masks/Normal-8776.png,True,G18245,train,d2f95ad3f12faa773f8e723ae9693c60,dd3dc2c31b91c543
Normal-8777.png,Normal,Normal/images/Normal-8777.png,Normal/masks/Normal-8777.png,True,G18246,train,11e637623048bebe4892f7bdeb7ca83d,c9bdc615974e504d
Normal-8778.png,Normal,Normal/images/Normal-8778.png,Normal/masks/Normal-8778.png,True,G18247,val,286666ba6b8592db6d4b171c3aa2b5b7,dd28d6531d444f1d
Normal-8779.png,Normal,Normal/images/Normal-8779.png,Normal/masks/Normal-8779.png,True,G18248,val,ff2759518db535b0ae37fdac0410fd07,d82fc20a0f5b3a4f
Normal-878.png,Normal,Normal/images/Normal-878.png,Normal/masks/Normal-878.png,True,G18249,test,4d73aa8847aafb8186bab595ff897d97,817dc61a9acb39d1
Normal-8780.png,Normal,Normal/images/Normal-8780.png,Normal/masks/Normal-8780.png,True,G18250,val,4a03614473d112831fa5f97dad186f60,883847971f451fda
Normal-8781.png,Normal,Normal/images/Normal-8781.png,Normal/masks/Normal-8781.png,True,G18251,test,85307b8e4cd422da8b5bc8f85854ab9d,dc2fc24f1d483d38
Normal-8782.png,Normal,Normal/images/Normal-8782.png,Normal/masks/Normal-8782.png,True,G18252,train,4d3e622c5b264005dbae18197b98cf7f,dd2c861f150e1e1f
Normal-8783.png,Normal,Normal/images/Normal-8783.png,Normal/masks/Normal-8783.png,True,G18253,val,48655e64d5f08a6fa448657050e7fdc4,8178ef869a1adb45
Normal-8784.png,Normal,Normal/images/Normal-8784.png,Normal/masks/Normal-8784.png,True,G18254,test,2086e5cb46033d0aa0cee8564a59b2b1,d87bc51616455e1d
Normal-8785.png,Normal,Normal/images/Normal-8785.png,Normal/masks/Normal-8785.png,True,G18255,train,a32daee2d0a5bd4a1aff69d89a6559c4,897f861a5d4559c3
Normal-8786.png,Normal,Normal/images/Normal-8786.png,Normal/masks/Normal-8786.png,True,G18256,train,f88053564839f6f0b6374ff158ba80ad,896906165e857b9f
Normal-8787.png,Normal,Normal/images/Normal-8787.png,Normal/masks/Normal-8787.png,True,G18257,train,2cc5c689e0e3613b2303b8564c3d3a3f,f92dd65a054a1b4d
Normal-8788.png,Normal,Normal/images/Normal-8788.png,Normal/masks/Normal-8788.png,True,G18258,train,c2407af155faa5c6221256875f756170,c03cc34b1e4d1f9d
Normal-8789.png,Normal,Normal/images/Normal-8789.png,Normal/masks/Normal-8789.png,True,G18259,train,ddb0a14a8ff9dacd5497bc289aecbb39,cc3dc25bd7412e0b
Normal-879.png,Normal,Normal/images/Normal-879.png,Normal/masks/Normal-879.png,True,G18260,train,e1a7543b647c40fb055dd020ccfc4145,8117c71ae89d7897
Normal-8790.png,Normal,Normal/images/Normal-8790.png,Normal/masks/Normal-8790.png,True,G18261,train,14c994160f75c21d2f074b67ba8edeea,996f851a053a39ed
Normal-8791.png,Normal,Normal/images/Normal-8791.png,Normal/masks/Normal-8791.png,True,G18262,val,4a05b92c1b9e81de253bf2ac81bc250c,883d571a551b3ac7
Normal-8792.png,Normal,Normal/images/Normal-8792.png,Normal/masks/Normal-8792.png,True,G18263,train,94817d170b195f03c0ddea45f8a16b0d,c03c0e074f7cea6d
Normal-8793.png,Normal,Normal/images/Normal-8793.png,Normal/masks/Normal-8793.png,True,G18264,test,bc109e96c34f30271ffc7e8c9207328d,887f87161f0d1f1c
Normal-8794.png,Normal,Normal/images/Normal-8794.png,Normal/masks/Normal-8794.png,True,G18265,train,babe879d9e972d1b48b9a0db936c8e97,897d86161a175e5b
Normal-8795.png,Normal,Normal/images/Normal-8795.png,Normal/masks/Normal-8795.png,True,G18266,train,769dfea240fbb355b2aa67322c76b2a9,d939c61e1c057b39
Normal-8796.png,Normal,Normal/images/Normal-8796.png,Normal/masks/Normal-8796.png,True,G18267,train,2af8bf125b0283c1a947d78ddf7aa37c,d82f421b0d4f393d
Normal-8797.png,Normal,Normal/images/Normal-8797.png,Normal/masks/Normal-8797.png,True,G18268,train,afcbb80f035b523f9ff3dc2ae27c5d8d,dc3dd25b59c21e18
Normal-8798.png,Normal,Normal/images/Normal-8798.png,Normal/masks/Normal-8798.png,True,G18269,train,41cf438525ee3cf0e8d10330aec51974,9979c71a5a451f38
Normal-8799.png,Normal,Normal/images/Normal-8799.png,Normal/masks/Normal-8799.png,True,G18270,train,b9b94519e168d3f4f22bbef4f75aa383,d42d42535d4b474f
Normal-88.png,Normal,Normal/images/Normal-88.png,Normal/masks/Normal-88.png,True,G18271,val,87260a894f906da6f9d0167af826ce68,d0bfcf1e19494896
Normal-880.png,Normal,Normal/images/Normal-880.png,Normal/masks/Normal-880.png,True,G18272,test,ad42b9c2f860e3a7b8da8b5140c2609d,d43ee71b8c4e1cc2
Normal-8800.png,Normal,Normal/images/Normal-8800.png,Normal/masks/Normal-8800.png,True,G18273,test,276c97fbc325c6325c98a39e9f215250,81680f5e5d4d78e9
Normal-8801.png,Normal,Normal/images/Normal-8801.png,Normal/masks/Normal-8801.png,True,G18274,train,cdfd941234460ea3e7b901bd00dd25bf,807b57161a1b79b9
Normal-8802.png,Normal,Normal/images/Normal-8802.png,Normal/masks/Normal-8802.png,True,G18275,train,fb6dfae4b05e9268334fd3705637a89d,9939861e1bc75f18
Normal-8803.png,Normal,Normal/images/Normal-8803.png,Normal/masks/Normal-8803.png,True,G18276,train,79fdef852ed19d0b94ee1d3d18a9d5f2,dd7cc6431b06493b
Normal-8804.png,Normal,Normal/images/Normal-8804.png,Normal/masks/Normal-8804.png,True,G18277,train,e5afba6ce6acd7c2b5900af313fd2bef,c17857861a9f2f89
Normal-8805.png,Normal,Normal/images/Normal-8805.png,Normal/masks/Normal-8805.png,True,G18278,train,5f52c2c9820e466df537b54608eb7505,91694ec6191f473b
Normal-8806.png,Normal,Normal/images/Normal-8806.png,Normal/masks/Normal-8806.png,True,G18279,test,e0ee2b23fbeff3fb7276e703e12e7b2d,9c6e82135b4d4f78
Normal-8807.png,Normal,Normal/images/Normal-8807.png,Normal/masks/Normal-8807.png,True,G18280,train,5da93acca528a3ee0b302383b9b49af3,c43dc3c30b0eeeca
Normal-8808.png,Normal,Normal/images/Normal-8808.png,Normal/masks/Normal-8808.png,True,G18281,train,bee122d6bde56121902101d10d2325b2,d919161a19a9ebb9
Normal-8809.png,Normal,Normal/images/Normal-8809.png,Normal/masks/Normal-8809.png,True,G18282,train,ba101b610096ac1b8af3b4a53c289b2b,997dd75a48061a6d
Normal-881.png,Normal,Normal/images/Normal-881.png,Normal/masks/Normal-881.png,True,G18283,train,73a92744666d79c9638c1c29db63bc3e,9056c29ecd34c6e7
Normal-8810.png,Normal,Normal/images/Normal-8810.png,Normal/masks/Normal-8810.png,True,G18284,train,1de81b71d7f308f5111aa75a609ed7dd,d938d61f460e9e4a
Normal-8811.png,Normal,Normal/images/Normal-8811.png,Normal/masks/Normal-8811.png,True,G18285,train,682ef3d7bf16c64e5327bdb44232bfab,dd28c65f15c64f90
Normal-8812.png,Normal,Normal/images/Normal-8812.png,Normal/masks/Normal-8812.png,True,G18286,train,6419ae272ba8068f355c5d89cd5a5ef7,817d461e19ebc9b8
Normal-8813.png,Normal,Normal/images/Normal-8813.png,Normal/masks/Normal-8813.png,True,G18287,val,5b062a5f1ea094913b26d4819fba57f9,d92bc6561e461b1e
Normal-8814.png,Normal,Normal/images/Normal-8814.png,Normal/masks/Normal-8814.png,True,G18288,val,6e2d32f3c4eb2f7c154a990132cf585b,dd2e52434f454e0f
Normal-8815.png,Normal,Normal/images/Normal-8815.png,Normal/masks/Normal-8815.png,True,G18289,test,e888180b44df350fd5e11d5839efee2e,dc39421fc82db0e7
Normal-8816.png,Normal,Normal/images/Normal-8816.png,Normal/masks/Normal-8816.png,True,G18290,train,04d57d6369272fd8fdfd5b1c6d9c0984,dd3dd04b92431bd2
Normal-8817.png,Normal,Normal/images/Normal-8817.png,Normal/masks/Normal-8817.png,True,G18291,val,f467b426127a2be3180c27dd817077a2,d13d07121d4f7c39
Normal-8818.png,Normal,Normal/images/Normal-8818.png,Normal/masks/Normal-8818.png,True,G18292,test,0e19cb41cecace88817f242d5b37872d,dc4c430b0e0f2f7d
Normal-8819.png,Normal,Normal/images/Normal-8819.png,Normal/masks/Normal-8819.png,True,G18293,test,68652a11551e6461e4b9a3c4e0a5bb78,d42dc21b194c2f3f
Normal-882.png,Normal,Normal/images/Normal-882.png,Normal/masks/Normal-882.png,True,G18294,train,ab72adeb75a17679831c1791200a2cb4,c0a7c33dc02b37c7
Normal-8820.png,Normal,Normal/images/Normal-8820.png,Normal/masks/Normal-8820.png,True,G18295,train,a2c13e20e2889bae62ca6c9ef7c829af,dd2d86171b8e1e1a
Normal-8821.png,Normal,Normal/images/Normal-8821.png,Normal/masks/Normal-8821.png,True,G18296,train,7c8a9930554fda8e435fe8a620455606,d92c461f1a6de839
Normal-8822.png,Normal,Normal/images/Normal-8822.png,Normal/masks/Normal-8822.png,True,G18297,train,98785e99c896149871f1cc5b7ef82a73,dd2cd30bd04b5bd0
Normal-8823.png,Normal,Normal/images/Normal-8823.png,Normal/masks/Normal-8823.png,True,G18298,train,a008fb27ce30ab2d1ea58577ccd90ac3,dd0a035d7d727270
Normal-8824.png,Normal,Normal/images/Normal-8824.png,Normal/masks/Normal-8824.png,True,G18299,train,f55a231e80337393e948fbbbea19b039,817f161859ef7a81
Normal-8825.png,Normal,Normal/images/Normal-8825.png,Normal/masks/Normal-8825.png,True,G18300,train,415b2c5ff360ea8703cbed193f6fabdc,897d871e144e1f55
Normal-8826.png,Normal,Normal/images/Normal-8826.png,Normal/masks/Normal-8826.png,True,G18301,train,886a2f92c71ce333c63aba0928fcf3fe,cc5e430b0d2aadaf
Normal-8827.png,Normal,Normal/images/Normal-8827.png,Normal/masks/Normal-8827.png,True,G18302,train,77dbdfd80aff87d54999947d1a9f1e62,993bc61b1aac3aa9
Normal-8828.png,Normal,Normal/images/Normal-8828.png,Normal/masks/Normal-8828.png,True,G18303,train,25a3f5a9d5000099ff3b058d4848ad29,9d2cc60bc3455acf
Normal-8829.png,Normal,Normal/images/Normal-8829.png,Normal/masks/Normal-8829.png,True,G18304,train,3c86180d3033618dafcfb0c55f0e0b05,dc2ec30b0e2f293d
Normal-883.png,Normal,Normal/images/Normal-883.png,Normal/masks/Normal-883.png,True,G18305,train,53084f699d55c6989058c6bae9c19c5e,8929c64adf4839f9
Normal-8830.png,Normal,Normal/images/Normal-8830.png,Normal/masks/Normal-8830.png,True,G18306,val,45366b586852900d23298cbc7e982198,9978061a1f063fbd
Normal-8831.png,Normal,Normal/images/Normal-8831.png,Normal/masks/Normal-8831.png,True,G18307,val,351f7908273964e3ed3244f481616ef6,cc7cd34b090e2f39
Normal-8832.png,Normal,Normal/images/Normal-8832.png,Normal/masks/Normal-8832.png,True,G18308,train,29ecf2bef1f6ad6c8482483136c24d0c,dd78c617ba09a6a4
Normal-8833.png,Normal,Normal/images/Normal-8833.png,Normal/masks/Normal-8833.png,True,G18309,train,28e1ee029c7524fbedef92ca50fa0b4a,c13846131b4fedb9
Normal-8834.png,Normal,Normal/images/Normal-8834.png,Normal/masks/Normal-8834.png,True,G18310,val,9aef6a06785f3ad04505b3f11cd2cdd0,dd2b42571a462f74
Normal-8835.png,Normal,Normal/images/Normal-8835.png,Normal/masks/Normal-8835.png,True,G18311,test,25feec01cc30a07fd769b95a6864f389,93601c531b5b739d
Normal-8836.png,Normal,Normal/images/Normal-8836.png,Normal/masks/Normal-8836.png,True,G18312,test,01e8f5744a21295f41832cbe3debbbb4,817b16195be9b8b1
Normal-8837.png,Normal,Normal/images/Normal-8837.png,Normal/masks/Normal-8837.png,True,G18313,test,ab520ea50d00b1c7ca14ac3f3d34eb86,9c7c434b4b0a3d4f
Normal-8838.png,Normal,Normal/images/Normal-8838.png,Normal/masks/Normal-8838.png,True,G18314,train,6ebcfbdefcbc4ddbef1878dc097b5e7c,d97fc6421bc44d19
Normal-8839.png,Normal,Normal/images/Normal-8839.png,Normal/masks/Normal-8839.png,True,G18315,train,aaf7e374dcac2d5ebdc202dcb7ab4b03,917a161a593b792b
Normal-884.png,Normal,Normal/images/Normal-884.png,Normal/masks/Normal-884.png,True,G18316,train,432786996a90f7fce4065adece486deb,d07e873b89ad68a4
Normal-8840.png,Normal,Normal/images/Normal-8840.png,Normal/masks/Normal-8840.png,True,G18317,train,062502f5bb146ae391ca7564967b716f,dd6d42c30b052f3b
Normal-8841.png,Normal,Normal/images/Normal-8841.png,Normal/masks/Normal-8841.png,True,G18318,train,5fdd33285293b7f38e05f43e53ef2a11,9d28961e1b071e5f
Normal-8842.png,Normal,Normal/images/Normal-8842.png,Normal/masks/Normal-8842.png,True,G18319,train,7370b2369e85faeeded9bab543d0043d,d97f841a1e4d1a39
Normal-8843.png,Normal,Normal/images/Normal-8843.png,Normal/masks/Normal-8843.png,True,G18320,train,fdac09e3c345773f896544e109945b55,897d861a195f39a9
Normal-8844.png,Normal,Normal/images/Normal-8844.png,Normal/masks/Normal-8844.png,True,G18321,train,a063979f06f74d1902ae781788e8d123,c03ccb4b0d4e2f3d
Normal-8845.png,Normal,Normal/images/Normal-8845.png,Normal/masks/Normal-8845.png,True,G18322,train,bcbf7ea2c9e27e475472b55f84866565,cd0a430f4d2efda2
Normal-8846.png,Normal,Normal/images/Normal-8846.png,Normal/masks/Normal-8846.png,True,G18323,train,b82524151ad158f2ecae873f110862ec,816ac41e581b7b7b
Normal-8847.png,Normal,Normal/images/Normal-8847.png,Normal/masks/Normal-8847.png,True,G18324,val,27146ad36d55ef8cc32ac223cc2e390a,dc2dd2535b42c74a
Normal-8848.png,Normal,Normal/images/Normal-8848.png,Normal/masks/Normal-8848.png,True,G18325,train,65d7ab0abd6a3d5d2787b3f7c0f4ba69,cc3dc20f0e6a3db4
Normal-8849.png,Normal,Normal/images/Normal-8849.png,Normal/masks/Normal-8849.png,True,G18326,val,1185bd14cc6718ca862687e207aaa6dc,d93b17161a115b5b
Normal-885.png,Normal,Normal/images/Normal-885.png,Normal/masks/Normal-885.png,True,G18327,val,a255a0c7024cb23e42c3a3894816fa00,dd2c034b0f4d3c7c
Normal-8850.png,Normal,Normal/images/Normal-8850.png,Normal/masks/Normal-8850.png,True,G18328,test,c4cd22fbc2a7c3186596e38aea564273,e969c75616565a1c
Normal-8851.png,Normal,Normal/images/Normal-8851.png,Normal/masks/Normal-8851.png,True,G18329,train,0f346a21ccc957d6c78d3ce54ef0e92e,cc3fd24f0e292da4
Normal-8852.png,Normal,Normal/images/Normal-8852.png,Normal/masks/Normal-8852.png,True,G18330,train,2107a0a57d3e4edc1306078342a65273,c02d47d33fc22f98
Normal-8853.png,Normal,Normal/images/Normal-8853.png,Normal/masks/Normal-8853.png,True,G18331,train,341961df1d9f3942db9619789fcce935,997c070a1b4d79b9
Normal-8854.png,Normal,Normal/images/Normal-8854.png,Normal/masks/Normal-8854.png,True,G18332,train,8dcdc0fd64af3df312c4ba32681d9b12,f979d65307431a0e
Normal-8855.png,Normal,Normal/images/Normal-8855.png,Normal/masks/Normal-8855.png,True,G18333,test,c715d5c00a4236a6d8a34d0a7abdf306,d9f5d654455b4158
Normal-8856.png,Normal,Normal/images/Normal-8856.png,Normal/masks/Normal-8856.png,True,G18334,val,e0d8567c3e43707fd72030e75ee9a9f6,dd2ed60a1f4c2a39
Normal-8857.png,Normal,Normal/images/Normal-8857.png,Normal/masks/Normal-8857.png,True,G18335,test,583e347ecf7405d3a884adac3943b28d,d929c61a554b3d4e
Normal-8858.png,Normal,Normal/images/Normal-8858.png,Normal/masks/Normal-8858.png,True,G18336,train,c938cc83c6ef04c550c9cb9c19a960b2,993e461a1c4df9b1
Normal-8859.png,Normal,Normal/images/Normal-8859.png,Normal/masks/Normal-8859.png,True,G18337,train,8b6ebdadaf23cfdbed616b0ff76d9d19,dd28d21b1f551b19
Normal-886.png,Normal,Normal/images/Normal-886.png,Normal/masks/Normal-886.png,True,G18338,train,b525e260f520fde475130e99924f4a6b,916dc65a8d4d19c7
Normal-8860.png,Normal,Normal/images/Normal-8860.png,Normal/masks/Normal-8860.png,True,G18339,train,7c809e69fa0f80f0ded89684be9f2883,d979565e5c491a49
Normal-8861.png,Normal,Normal/images/Normal-8861.png,Normal/masks/Normal-8861.png,True,G18340,train,1037ee8bff45814f59f250d595647f16,d57fc20a1e1e1b1c
Normal-8862.png,Normal,Normal/images/Normal-8862.png,Normal/masks/Normal-8862.png,True,G18341,train,014577397927688ff8876e5c6066be66,dd2ac217d5425f19
Normal-8863.png,Normal,Normal/images/Normal-8863.png,Normal/masks/Normal-8863.png,True,G18342,train,a552074df3f65f7d3b09869af5a1ce68,81ed461b1f4f1917
Normal-8864.png,Normal,Normal/images/Normal-8864.png,Normal/masks/Normal-8864.png,True,G18343,train,649b42528e97b659f38f4d1855945c9b,81791c1a5b6b78e3
Normal-8865.png,Normal,Normal/images/Normal-8865.png,Normal/masks/Normal-8865.png,True,G18344,train,f7ca840a15f52dfc7cb1fdd1b1cdee0f,816bd71ad45f105e
Normal-8866.png,Normal,Normal/images/Normal-8866.png,Normal/masks/Normal-8866.png,True,G18345,train,037d47691b28244f6e527ae68678a5f2,897f061e171a5a7a
Normal-8867.png,Normal,Normal/images/Normal-8867.png,Normal/masks/Normal-8867.png,True,G18346,train,d101ecd0953b97736341879610037871,c979861a594b1f4d
Normal-8868.png,Normal,Normal/images/Normal-8868.png,Normal/masks/Normal-8868.png,True,G18347,train,c10729d053899b1874a2c04a1c734ded,807bc7841c1eda5f
Normal-8869.png,Normal,Normal/images/Normal-8869.png,Normal/masks/Normal-8869.png,True,G18348,val,beea68e995ef45e3ccfa4f3f39b68f71,992846571e571e1f
Normal-887.png,Normal,Normal/images/Normal-887.png,Normal/masks/Normal-887.png,True,G18349,train,167843db9fcd520f73a70124d0a8fe4f,816ec75acdc969c1
Normal-8870.png,Normal,Normal/images/Normal-8870.png,Normal/masks/Normal-8870.png,True,G18350,val,61c437e542e3785951d5e1683f3b6758,dd2dc2d30bc08b9d
Normal-8871.png,Normal,Normal/images/Normal-8871.png,Normal/masks/Normal-8871.png,True,G18351,test,6761c2a3a306a0f2b0c752169188ce4a,c42c4bd34d8d2f9c
Normal-8872.png,Normal,Normal/images/Normal-8872.png,Normal/masks/Normal-8872.png,True,G18352,train,4f314cbf8f3951140434d544dc6aea29,d12b86171e5e1b1e
Normal-8873.png,Normal,Normal/images/Normal-8873.png,Normal/masks/Normal-8873.png,True,G18353,test,894775b9495f6e4ac33194b39eb60882,dd2dc61a194e196d
Normal-8874.png,Normal,Normal/images/Normal-8874.png,Normal/masks/Normal-8874.png,True,G18354,test,70e2a213b0c806b3b39e6a460eb31d56,9979071e5e571859
Normal-8875.png,Normal,Normal/images/Normal-8875.png,Normal/masks/Normal-8875.png,True,G18355,train,55171d827e17bf83e5f4be1f1004a47b,dd66c3490d4e4e47
Normal-8876.png,Normal,Normal/images/Normal-8876.png,Normal/masks/Normal-8876.png,True,G18356,train,ec0b3706c056fd0aeceb5273abfd8a59,cc7f834b0f0b0d4e
Normal-8877.png,Normal,Normal/images/Normal-8877.png,Normal/masks/Normal-8877.png,True,G18357,train,ca456644acb4e7b4703dfc2ce17423a3,d939d7165e090f1a
Normal-8878.png,Normal,Normal/images/Normal-8878.png,Normal/masks/Normal-8878.png,True,G18358,train,daac54b5dd197b0141deb1a1fb504a10,b92dc61a097d786c
Normal-8879.png,Normal,Normal/images/Normal-8879.png,Normal/masks/Normal-8879.png,True,G18359,train,ff8ca4d22c31b614ba7b00a2e78e7c29,c17f56161a467b29
Normal-888.png,Normal,Normal/images/Normal-888.png,Normal/masks/Normal-888.png,True,G18360,train,4d7c59ff0d7b273f9d87aaf3aec23cb9,947dc71a998d1ccc
Normal-8880.png,Normal,Normal/images/Normal-8880.png,Normal/masks/Normal-8880.png,True,G18361,train,6a5c6c268b7d8c745673390171c6138a,b83d87131b4f585a
Normal-8881.png,Normal,Normal/images/Normal-8881.png,Normal/masks/Normal-8881.png,True,G18362,train,a6566aae811d0be789e3a2a5635c2031,dd3dc61f1a444d38
Normal-8882.png,Normal,Normal/images/Normal-8882.png,Normal/masks/Normal-8882.png,True,G18363,train,1417cdaf58bbd1a5ec59a966649752cf,8159de021b191eff
Normal-8883.png,Normal,Normal/images/Normal-8883.png,Normal/masks/Normal-8883.png,True,G18364,val,5ebb39f57471015ba5d80ec000b425f0,d13d461e1a1e5f19
Normal-8884.png,Normal,Normal/images/Normal-8884.png,Normal/masks/Normal-8884.png,True,G18365,val,7f53ff99ea0adb099e853f4388841cea,dd2e071b0d7c0a7c
Normal-8885.png,Normal,Normal/images/Normal-8885.png,Normal/masks/Normal-8885.png,True,G18366,train,1138f5ce92308cc7bfc9a21e5e6044db,9979c61a194e1fb1
Normal-8886.png,Normal,Normal/images/Normal-8886.png,Normal/masks/Normal-8886.png,True,G18367,train,164eb594e5d4640b74b5a9e4ea2b1762,d02dc6021b7d6f3c
Normal-8887.png,Normal,Normal/images/Normal-8887.png,Normal/masks/Normal-8887.png,True,G18368,train,324482642f3f86190e45c16bd4b26060,d53c02c35f5cae86
Normal-8888.png,Normal,Normal/images/Normal-8888.png,Normal/masks/Normal-8888.png,True,G18369,val,ebd602cb1f0a8ca026e6ea33750f7146,d02947561e5f4f1c
Normal-8889.png,Normal,Normal/images/Normal-8889.png,Normal/masks/Normal-8889.png,True,G18370,train,25d37019ee20bed5647cf08a1f38cb53,d42e52534b4b4f4e
Normal-889.png,Normal,Normal/images/Normal-889.png,Normal/masks/Normal-889.png,True,G18371,test,794de035562d847ef6433661f512efe6,d42f4f0b4d4569ac
Normal-8890.png,Normal,Normal/images/Normal-8890.png,Normal/masks/Normal-8890.png,True,G18372,test,9dd962e8f7ed2bb7a253292a6a6087e0,9d691c1e7b4ae123
Normal-8891.png,Normal,Normal/images/Normal-8891.png,Normal/masks/Normal-8891.png,True,G18373,test,0ff233397ad9be4cd046bfba5dece72c,982d870b0d4f3d9d
Normal-8892.png,Normal,Normal/images/Normal-8892.png,Normal/masks/Normal-8892.png,True,G18374,train,803d92bd07303ff79e545b2ec8d4b83b,917bd71a14571a1e
Normal-8893.png,Normal,Normal/images/Normal-8893.png,Normal/masks/Normal-8893.png,True,G18375,train,e982760e3f2988779edc4afbc08fbc5d,8169c7c65a5dcc59
Normal-8894.png,Normal,Normal/images/Normal-8894.png,Normal/masks/Normal-8894.png,True,G18376,train,c59616da50c1346d1ce3c1895dd1d168,8179071e1f5e1b5c
Normal-8895.png,Normal,Normal/images/Normal-8895.png,Normal/masks/Normal-8895.png,True,G18377,train,7682c2311e649ac27d594acf9b362f3e,dd2a131d5c1855db
Normal-8896.png,Normal,Normal/images/Normal-8896.png,Normal/masks/Normal-8896.png,True,G18378,train,165fa3600321428a20d3110ee5b7bada,dd2f86071c4c6f4c
Normal-8897.png,Normal,Normal/images/Normal-8897.png,Normal/masks/Normal-8897.png,True,G18379,train,fe7155d5f60d311256653a5b2ecf7c34,d93906061e1d6b7b
Normal-8898.png,Normal,Normal/images/Normal-8898.png,Normal/masks/Normal-8898.png,True,G18380,train,7e7b6f4b38ca658dae7c64d7b8f5178f,917987861a97c71b
Normal-8899.png,Normal,Normal/images/Normal-8899.png,Normal/masks/Normal-8899.png,True,G18381,train,750cb1ec750fc70e04aa71e0421a8009,cc2a530b5b6d2a3d
Normal-89.png,Normal,Normal/images/Normal-89.png,Normal/masks/Normal-89.png,True,G18382,train,0c921496d974ff870ec9ca499684d7c5,9174c63aad6d386c
Normal-890.png,Normal,Normal/images/Normal-890.png,Normal/masks/Normal-890.png,True,G18383,train,65fbef21acca2c84711e65062c87ac9b,d1e5c71ac96939c1
Normal-8900.png,Normal,Normal/images/Normal-8900.png,Normal/masks/Normal-8900.png,True,G18384,test,16c50523f3142e0120d4cf90c0a470c6,c82d9242bf12b779
Normal-8901.png,Normal,Normal/images/Normal-8901.png,Normal/masks/Normal-8901.png,True,G18385,train,18996b966b6d1f5fc04d0dfff6bb289e,cc3cc30b870b1fcb
Normal-8902.png,Normal,Normal/images/Normal-8902.png,Normal/masks/Normal-8902.png,True,G18386,train,da10497ec3b0acaff66f074d70d27bbd,d93c0e18ade5b8c5
Normal-8903.png,Normal,Normal/images/Normal-8903.png,Normal/masks/Normal-8903.png,True,G18387,train,fdf2390afd287b0462984d8146591c55,a82c871b1d4d5d5d
Normal-8904.png,Normal,Normal/images/Normal-8904.png,Normal/masks/Normal-8904.png,True,G18388,train,b9fc8396983b751cd85e56d6664dc411,9929865b194e7b39
Normal-8905.png,Normal,Normal/images/Normal-8905.png,Normal/masks/Normal-8905.png,True,G18389,val,11bd59d60d5465873cb93ffdc0adf158,92639c505c587b7e
Normal-8906.png,Normal,Normal/images/Normal-8906.png,Normal/masks/Normal-8906.png,True,G18390,val,955e552b641b30dfd3e6a8585fe7f898,cc3cc20f8e4b1dce
Normal-8907.png,Normal,Normal/images/Normal-8907.png,Normal/masks/Normal-8907.png,True,G18391,test,54737c80ab8b647fed99ca53a2581de8,9928161a1e5f5d5e
Normal-8908.png,Normal,Normal/images/Normal-8908.png,Normal/masks/Normal-8908.png,True,G18392,val,1cd1f12047d11be36013182e70ce6adb,dc6dc20b8d08bee5
Normal-8909.png,Normal,Normal/images/Normal-8909.png,Normal/masks/Normal-8909.png,True,G18393,test,0c8b568ef6a0c229d4971216f9288cc1,dc3dc342fd08ae2c
Normal-891.png,Normal,Normal/images/Normal-891.png,Normal/masks/Normal-891.png,True,G18394,train,389055d885871d247ce2d3c5c6f71b8b,952c0e5b7be868e1
Normal-8910.png,Normal,Normal/images/Normal-8910.png,Normal/masks/Normal-8910.png,True,G18395,train,c2f8bc12ce6d2bb6b81080b1617ccbed,dd2b46871b0cee2a
Normal-8911.png,Normal,Normal/images/Normal-8911.png,Normal/masks/Normal-8911.png,True,G18396,train,55ea5f4dea38140fc855364c0036d235,d96c861b956a38ec
Normal-8912.png,Normal,Normal/images/Normal-8912.png,Normal/masks/Normal-8912.png,True,G18397,train,18123f8da3126a0f47d5b0d83f752911,dd2dc2071b474e1d
Normal-8913.png,Normal,Normal/images/Normal-8913.png,Normal/masks/Normal-8913.png,True,G18398,train,72ce9766c7fcff35a30ba6b21647971e,91281f1b1a1bbb9e
Normal-8914.png,Normal,Normal/images/Normal-8914.png,Normal/masks/Normal-8914.png,True,G18399,train,c82f535bdb1140f1aab20c6a7a712f25,88fd871a851e1f0f
Normal-8915.png,Normal,Normal/images/Normal-8915.png,Normal/masks/Normal-8915.png,True,G18400,test,5c235ea80e66ab046735f7aeefa62d11,c03dc3cb0b0b8f2f
Normal-8916.png,Normal,Normal/images/Normal-8916.png,Normal/masks/Normal-8916.png,True,G18401,train,da987e2bf156f595916c82b9be6f013a,807947861e1d6f5b
Normal-8917.png,Normal,Normal/images/Normal-8917.png,Normal/masks/Normal-8917.png,True,G18402,train,b640c1e9cca33ddf1ab796eb11ef54d4,c03d0bcb8c8cbcf3
Normal-8918.png,Normal,Normal/images/Normal-8918.png,Normal/masks/Normal-8918.png,True,G18403,train,4bc7c17d373221131365ffb011bb4e09,9ce0475b0e1f2a8f
Normal-8919.png,Normal,Normal/images/Normal-8919.png,Normal/masks/Normal-8919.png,True,G18404,train,80c57853ad3925ee2f4b0da0c2e2aecf,884d071a5d6f78e6
Normal-892.png,Normal,Normal/images/Normal-892.png,Normal/masks/Normal-892.png,True,G18405,val,964ebcde695956a821bf286b2819f310,d1a9c21eda793869
Normal-8920.png,Normal,Normal/images/Normal-8920.png,Normal/masks/Normal-8920.png,True,G18406,train,093577397249d88241f51c41042287fd,917906869a9becea
Normal-8921.png,Normal,Normal/images/Normal-8921.png,Normal/masks/Normal-8921.png,True,G18407,train,3bee3a7bcb845a19fe451c72557b2646,c139c6461b1d6e5b
Normal-8922.png,Normal,Normal/images/Normal-8922.png,Normal/masks/Normal-8922.png,True,G18408,train,86c3fac4ec2d2be64c737b7df824b45f,896d061a6bcdf8c5
Normal-8923.png,Normal,Normal/images/Normal-8923.png,Normal/masks/Normal-8923.png,True,G18409,train,2d65dbc48d394fac6e816f2f50e3b5bf,d929861a1d4f3b1d
Normal-8924.png,Normal,Normal/images/Normal-8924.png,Normal/masks/Normal-8924.png,True,G18410,test,49ab18fda1fae0308ab02d1d47d69ea2,d93dd6431f464a19
Normal-8925.png,Normal,Normal/images/Normal-8925.png,Normal/masks/Normal-8925.png,True,G18411,val,d8d729a1b1f1a58e915bb0bc54438fc4,9d6cce1a144d706f
Normal-8926.png,Normal,Normal/images/Normal-8926.png,Normal/masks/Normal-8926.png,True,G18412,val,0962a19244f4c05f424268f4478da801,9979851e1d4e1b99
Normal-8927.png,Normal,Normal/images/Normal-8927.png,Normal/masks/Normal-8927.png,True,G18413,test,e650fff1ce5f62483745b2b23234caee,993b06161e555f5a
Normal-8928.png,Normal,Normal/images/Normal-8928.png,Normal/masks/Normal-8928.png,True,G18414,test,1cd4ae246bfd2ed1edd271e77eda12e0,dc2b530d4d0d7da8
Normal-8929.png,Normal,Normal/images/Normal-8929.png,Normal/masks/Normal-8929.png,True,G18415,train,5c745d905fddcac6091cd925d34be3fe,81798e5e195e2a79
Normal-893.png,Normal,Normal/images/Normal-893.png,Normal/masks/Normal-893.png,True,G18416,train,fcd2daf5a21b66a43b30807480987e7d,813a565e7ee82939
Normal-8930.png,Normal,Normal/images/Normal-8930.png,Normal/masks/Normal-8930.png,True,G18417,train,18a101f54fbc6e4198d8cbcba9c57aad,887b07861e9e5a7a
Normal-8931.png,Normal,Normal/images/Normal-8931.png,Normal/masks/Normal-8931.png,True,G18418,train,dca2a5bedf6ecdb437834e3e9b3a8a45,d939d64a1f044b1f
Normal-8932.png,Normal,Normal/images/Normal-8932.png,Normal/masks/Normal-8932.png,True,G18419,train,830e07b215e92715e89f4e911a37dc88,880f461e4d4f78eb
Normal-8933.png,Normal,Normal/images/Normal-8933.png,Normal/masks/Normal-8933.png,True,G18420,train,9d7d6e99b3b697752e6d8013167a8166,817b05171a1f7a7a
Normal-8934.png,Normal,Normal/images/Normal-8934.png,Normal/masks/Normal-8934.png,True,G18421,train,b3277aecfc01cb93dbb4c6a8da8a66ae,9919065a1b787be9
Normal-8935.png,Normal,Normal/images/Normal-8935.png,Normal/masks/Normal-8935.png,True,G18422,train,27e43522cef5267961eba6863ba43abc,887bf7061e9c145e
Normal-8936.png,Normal,Normal/images/Normal-8936.png,Normal/masks/Normal-8936.png,True,G18423,train,3eafc5f09d3efecd27c48f2b3e8fc265,986d871e855e1a4f
Normal-8937.png,Normal,Normal/images/Normal-8937.png,Normal/masks/Normal-8937.png,True,G18424,train,f7f3f39ca249d6ed049713eb39a3022d,91691e1b597c78a5
Normal-8938.png,Normal,Normal/images/Normal-8938.png,Normal/masks/Normal-8938.png,True,G18425,val,af5c201b7603227f25fe193462472d5f,cc3dc215524f3b69
Normal-8939.png,Normal,Normal/images/Normal-8939.png,Normal/masks/Normal-8939.png,True,G18426,train,13c157bed61cd27cd8491e9f46c1ee44,bc2dc1921bc31f1e
Normal-894.png,Normal,Normal/images/Normal-894.png,Normal/masks/Normal-894.png,True,G18427,train,a37401a3e84634fce208f417294fe3a9,912bc21ada7939d9
Normal-8940.png,Normal,Normal/images/Normal-8940.png,Normal/masks/Normal-8940.png,True,G18428,train,a1a0111858b0a54318830062eabd49ea,d03c4f031c2dbe3d
Normal-8941.png,Normal,Normal/images/Normal-8941.png,Normal/masks/Normal-8941.png,True,G18429,train,809992e5a141144252112f9bba561bc7,8c2cc4535b555e5e
Normal-8942.png,Normal,Normal/images/Normal-8942.png,Normal/masks/Normal-8942.png,True,G18430,val,71dec210b1c6abdfab047132c7ae759f,dc2f534b4d4d0c4e
Normal-8943.png,Normal,Normal/images/Normal-8943.png,Normal/masks/Normal-8943.png,True,G18431,val,42a7f389dca17a9b8ca21270d069cea8,886f861e1b1f1e1d
Normal-8944.png,Normal,Normal/images/Normal-8944.png,Normal/masks/Normal-8944.png,True,G18432,train,3da149020020e981e1dc8542423683ee,8c0e434f6c7e66e8
Normal-8945.png,Normal,Normal/images/Normal-8945.png,Normal/masks/Normal-8945.png,True,G18433,test,27612473a46a8ea1477ca81639313f59,d865d656d45ad44a
Normal-8946.png,Normal,Normal/images/Normal-8946.png,Normal/masks/Normal-8946.png,True,G18434,test,f0e85c7cd988083d41d386478f924a39,dd2cd24b0f4a0f1d
Normal-8947.png,Normal,Normal/images/Normal-8947.png,Normal/masks/Normal-8947.png,True,G18435,train,8f5253eb37ca49099f27fce5e4c3d3b3,c93d420f19ee2db1
Normal-8948.png,Normal,Normal/images/Normal-8948.png,Normal/masks/Normal-8948.png,True,G18436,train,5a53ad74d7f7e36f7fe893443ab83a7d,997fc692154d185d
Normal-8949.png,Normal,Normal/images/Normal-8949.png,Normal/masks/Normal-8949.png,True,G18437,val,8d577c314f4ca1447a47bbac3cfe2665,997806161b5f5979
Normal-895.png,Normal,Normal/images/Normal-895.png,Normal/masks/Normal-895.png,True,G18438,test,24d09278bc3a0f60b617e540472df8dd,d47f871f8d0d0c86
Normal-8950.png,Normal,Normal/images/Normal-8950.png,Normal/masks/Normal-8950.png,True,G18439,val,4da59f2a0d0b7719d67baaefbc1c6bbc,dd2e535b0b4f0a0b
Normal-8951.png,Normal,Normal/images/Normal-8951.png,Normal/masks/Normal-8951.png,True,G18440,test,bf75c3b5ff3783e3fa94b951ce3cbc1a,d4bb421d124f1ccf
Normal-8952.png,Normal,Normal/images/Normal-8952.png,Normal/masks/Normal-8952.png,True,G18441,train,50f79f36d29fcfa8dd6ffc703e456e53,9d2c420b1fc43ebd
Normal-8953.png,Normal,Normal/images/Normal-8953.png,Normal/masks/Normal-8953.png,True,G18442,train,f179c9413f972ad9209ecbb87273f9a0,dd30d20ffa4a5a4a
Normal-8954.png,Normal,Normal/images/Normal-8954.png,Normal/masks/Normal-8954.png,True,G18443,train,ce2d05f91c1e4fc3c483900bdde85812,dd7fc2035a4c1b19
Normal-8955.png,Normal,Normal/images/Normal-8955.png,Normal/masks/Normal-8955.png,True,G18444,train,8400cdc21d02e06604d5383c8667293b,917d461a196f3a69
Normal-8956.png,Normal,Normal/images/Normal-8956.png,Normal/masks/Normal-8956.png,True,G18445,train,fcfeeb3697929fc677a6601fdd84df8c,817b4712134f1e5f
Normal-8957.png,Normal,Normal/images/Normal-8957.png,Normal/masks/Normal-8957.png,True,G18446,train,90b8d633dfe5a963ccf1b3bcb75a6e46,dd2cc01e93471e4f
Normal-8958.png,Normal,Normal/images/Normal-8958.png,Normal/masks/Normal-8958.png,True,G18447,train,c3920cf52ddebb8b7c4b9faae172501a,c13956521aad6b3d
Normal-8959.png,Normal,Normal/images/Normal-8959.png,Normal/masks/Normal-8959.png,True,G18448,train,c48654e6c8fe0d37fa8b50c4449fa098,dd2cd2530b554b1d
Normal-896.png,Normal,Normal/images/Normal-896.png,Normal/masks/Normal-896.png,True,G18449,train,b26a0625de48d8df5e212ea2e52bee75,d56f870b8d2c6c64
Normal-8960.png,Normal,Normal/images/Normal-8960.png,Normal/masks/Normal-8960.png,True,G18450,train,8176c86d307cde43feb793029d0258fb,d97fc20b1c4a1b3c
Normal-8961.png,Normal,Normal/images/Normal-8961.png,Normal/masks/Normal-8961.png,True,G18451,train,5dd9a72a370ecfa298bc1cc3fac5fd6d,897f061a457a786b
Normal-8962.png,Normal,Normal/images/Normal-8962.png,Normal/masks/Normal-8962.png,True,G18452,val,1f65066d561730764b8edd85d4cbf6a5,b96f061a461f505f
Normal-8963.png,Normal,Normal/images/Normal-8963.png,Normal/masks/Normal-8963.png,True,G18453,test,57c9f07ab3a7cb946425fd727dedd35b,8979069a5e957a99
Normal-8964.png,Normal,Normal/images/Normal-8964.png,Normal/masks/Normal-8964.png,True,G18454,test,f237a20143fcd95ca536b678cf29288c,9b615e1e1a3999e4
Normal-8965.png,Normal,Normal/images/Normal-8965.png,Normal/masks/Normal-8965.png,True,G18455,train,fccc29c2341bd629722162e358c23312,997fc61e844a3cc3
Normal-8966.png,Normal,Normal/images/Normal-8966.png,Normal/masks/Normal-8966.png,True,G18456,train,3c5b821d81b6e7517aaa3a6cd4bf7c2d,817b171a1e5e1a3e
Normal-8967.png,Normal,Normal/images/Normal-8967.png,Normal/masks/Normal-8967.png,True,G18457,test,654ebe738d6b3bd773ac11d41af03a83,897f045e1a185f5b
Normal-8968.png,Normal,Normal/images/Normal-8968.png,Normal/masks/Normal-8968.png,True,G18458,train,30a7d76b0680c7d8acf9185a40c1b67b,c03dfa435f5c904b
Normal-8969.png,Normal,Normal/images/Normal-8969.png,Normal/masks/Normal-8969.png,True,G18459,train,57b426b6f31834a11f4a2eb885479d53,894f8e194df2eaa2
Normal-897.png,Normal,Normal/images/Normal-897.png,Normal/masks/Normal-897.png,True,G18460,train,bcd2335fff50005f32f1af63408c4f2a,917fd61a1519394d
Normal-8970.png,Normal,Normal/images/Normal-8970.png,Normal/masks/Normal-8970.png,True,G18461,train,229a11a7a0b73c329659ec2cfe6f4002,c07d4b093dcc3dd4
Normal-8971.png,Normal,Normal/images/Normal-8971.png,Normal/masks/Normal-8971.png,True,G18462,train,cce999ea12921aef6b65009ff57eaf05,9c0e434f0cfe12bb
Normal-8972.png,Normal,Normal/images/Normal-8972.png,Normal/masks/Normal-8972.png,True,G18463,train,5aafbe8be266cd80e2b26cbde909ba74,816976d6d65a924e
Normal-8973.png,Normal,Normal/images/Normal-8973.png,Normal/masks/Normal-8973.png,True,G18464,train,e33bd1698efcd319103e40395686f122,dd2e425d0a6f1ccc
Normal-8974.png,Normal,Normal/images/Normal-8974.png,Normal/masks/Normal-8974.png,True,G18465,train,1ea095d9d7faee8b257d2d4334293a21,dd78430b554b1d4d
Normal-8975.png,Normal,Normal/images/Normal-8975.png,Normal/masks/Normal-8975.png,True,G18466,val,1a907a8a6c75e86b5f8bc0dd1d0859a3,dc44434b4b0f2f2f
Normal-8976.png,Normal,Normal/images/Normal-8976.png,Normal/masks/Normal-8976.png,True,G18467,test,1f7f8593866ad244dad3202adbf8d603,c93cc71b154d38e9
Normal-8977.png,Normal,Normal/images/Normal-8977.png,Normal/masks/Normal-8977.png,True,G18468,train,a871525aef94b5af7458e42d8f97b515,99788643195f875e
Normal-8978.png,Normal,Normal/images/Normal-8978.png,Normal/masks/Normal-8978.png,True,G18469,train,eaea083e80fd5d0b414c87a2fd06b288,987905165e4f727a
Normal-8979.png,Normal,Normal/images/Normal-8979.png,Normal/masks/Normal-8979.png,True,G18470,train,7dedcbfcb7e90a801737658ba1ea2231,952ec35b0d4f1a5c
Normal-898.png,Normal,Normal/images/Normal-898.png,Normal/masks/Normal-898.png,True,G18471,train,27202f506855982b658397b7d07ac4f7,91fbc65a5d491859
Normal-8980.png,Normal,Normal/images/Normal-8980.png,Normal/masks/Normal-8980.png,True,G18472,test,96ce73f2d9ac4a5298f50c6e255da73b,c929845b9d471b5d
Normal-8981.png,Normal,Normal/images/Normal-8981.png,Normal/masks/Normal-8981.png,True,G18473,train,c772ba200a695c214e24e50b0a88d522,917947861e971e95
Normal-8982.png,Normal,Normal/images/Normal-8982.png,Normal/masks/Normal-8982.png,True,G18474,train,94469ec4ad7d1d0370b46e734ef59eb4,913b5e161e1e4f1a
Normal-8983.png,Normal,Normal/images/Normal-8983.png,Normal/masks/Normal-8983.png,True,G18475,train,90f53997f8c4f4d2411c1fdc13fbbfcd,9921071e190f7f3b
Normal-8984.png,Normal,Normal/images/Normal-8984.png,Normal/masks/Normal-8984.png,True,G18476,val,9600e1e5641e271314ca1a3283686c86,992d4613194d78bf
Normal-8985.png,Normal,Normal/images/Normal-8985.png,Normal/masks/Normal-8985.png,True,G18477,val,d4009c75f1998bd081dca8f937cdfa31,9979c61a1d4d1a3d
Normal-8986.png,Normal,Normal/images/Normal-8986.png,Normal/masks/Normal-8986.png,True,G18478,train,d0f04d12c6f4f0e8deef68ce2259c6e1,9968861a4f1a5b5f
Normal-8987.png,Normal,Normal/images/Normal-8987.png,Normal/masks/Normal-8987.png,True,G18479,train,34dd5b1e9254d4aa8143262f7367b5f5,8179c61e1b5d4979
Normal-8988.png,Normal,Normal/images/Normal-8988.png,Normal/masks/Normal-8988.png,True,G18480,train,aecf4e8b496edcf0ac3070ccae8628c7,997946161e5b195b
Normal-8989.png,Normal,Normal/images/Normal-8989.png,Normal/masks/Normal-8989.png,True,G18481,train,092beaa6223d6cf8d7bb835d19379fbb,dd2e425f064b194f
Normal-899.png,Normal,Normal/images/Normal-899.png,Normal/masks/Normal-899.png,True,G18482,val,2f719862fc6e566bfe72e1af9378a2ca,817f071a124f7c3d
Normal-8990.png,Normal,Normal/images/Normal-8990.png,Normal/masks/Normal-8990.png,True,G18483,test,811bf4005200ad3964e996729035e7af,ca2d4b0a2d2f1f95
Normal-8991.png,Normal,Normal/images/Normal-8991.png,Normal/masks/Normal-8991.png,True,G18484,train,185c96f68d8371a67ab78c244c50b3ab,897986165a5f1b1b
Normal-8992.png,Normal,Normal/images/Normal-8992.png,Normal/masks/Normal-8992.png,True,G18485,train,1549ac145e36da0784c689635ae9ea24,d52d42875a0a6f5e
Normal-8993.png,Normal,Normal/images/Normal-8993.png,Normal/masks/Normal-8993.png,True,G18486,train,f56a2b448240b8c46b3b831a84130323,85785762a19f995a
Normal-8994.png,Normal,Normal/images/Normal-8994.png,Normal/masks/Normal-8994.png,True,G18487,train,af54ea9bab04f71b0c067a0f4cb40a7b,997b461a5d49389d
Normal-8995.png,Normal,Normal/images/Normal-8995.png,Normal/masks/Normal-8995.png,True,G18488,test,8d69705f2426342fb430032fab5b408d,913a821d0f1a5f5f
Normal-8996.png,Normal,Normal/images/Normal-8996.png,Normal/masks/Normal-8996.png,True,G18489,train,ee640a26c3cb6e11728ebf2291a6272e,896d861a4b5f784b
Normal-8997.png,Normal,Normal/images/Normal-8997.png,Normal/masks/Normal-8997.png,True,G18490,train,d2e55ba616eeff15835506a7f9ee94ec,d13956471e1acf0e
Normal-8998.png,Normal,Normal/images/Normal-8998.png,Normal/masks/Normal-8998.png,True,G18491,train,568c7e2c9f0a9b2137c28b13b45a953a,d43dc6431d0eed0b
Normal-8999.png,Normal,Normal/images/Normal-8999.png,Normal/masks/Normal-8999.png,True,G18492,val,4066ae41e4838f9a89dc971197991142,88fd8612574f1f15
Normal-9.png,Normal,Normal/images/Normal-9.png,Normal/masks/Normal-9.png,True,G18493,val,8d08fe25cd6032d5a452c683a5c9b599,916806471b5ded7a
Normal-90.png,Normal,Normal/images/Normal-90.png,Normal/masks/Normal-90.png,True,G18494,test,98ff7d775e787f4339ade2db54e3b4d6,d46ec71fcd052cc4
Normal-900.png,Normal,Normal/images/Normal-900.png,Normal/masks/Normal-900.png,True,G18495,train,e16dc954fb5bb9fd1863273aa0ea5d70,90abc34b3cea39b8
Normal-9000.png,Normal,Normal/images/Normal-9000.png,Normal/masks/Normal-9000.png,True,G18496,train,49db780c36a8014b83ce8da47d0658b6,cd2c420b4d6d3eec
Normal-9001.png,Normal,Normal/images/Normal-9001.png,Normal/masks/Normal-9001.png,True,G18497,train,49c1a68dc43d50fb7cf2d0ad75d75933,d62ec1494a2d6f0f
Normal-9002.png,Normal,Normal/images/Normal-9002.png,Normal/masks/Normal-9002.png,True,G18498,train,7a5cf058b44d42e1161da2586fed8e10,cc05430f4e2f4f2f
Normal-9003.png,Normal,Normal/images/Normal-9003.png,Normal/masks/Normal-9003.png,True,G18499,train,189175ca7c5ea3b72431bcc500d01b77,985f171a69e863e4
Normal-9004.png,Normal,Normal/images/Normal-9004.png,Normal/masks/Normal-9004.png,True,G18500,train,3ffcee61d8b8c727a4832b2d559798e9,8979c71a15ce1e95
Normal-9005.png,Normal,Normal/images/Normal-9005.png,Normal/masks/Normal-9005.png,True,G18501,train,2d059eeeeefec5965dd129008bc0ca94,817816595b6ef8c3
Normal-9006.png,Normal,Normal/images/Normal-9006.png,Normal/masks/Normal-9006.png,True,G18502,train,3df1fccc2dd9fd29ac98448007f929d8,dc3cc24b19862fb9
Normal-9007.png,Normal,Normal/images/Normal-9007.png,Normal/masks/Normal-9007.png,True,G18503,train,60197ca2cf3e3b2ac800b681373f5f9f,993c421b4d0ebabd
Normal-9008.png,Normal,Normal/images/Normal-9008.png,Normal/masks/Normal-9008.png,True,G18504,train,601b5da8365033210a401e18d2e9b290,d13d06161b4f693d
Normal-9009.png,Normal,Normal/images/Normal-9009.png,Normal/masks/Normal-9009.png,True,G18505,train,cd35cba96133806c34d266e123df0860,d4bc4751d343d343
Normal-901.png,Normal,Normal/images/Normal-901.png,Normal/masks/Normal-901.png,True,G18506,train,59aff61c683aff6c48586df502d423ac,9828c75e5f591d4c
Normal-9010.png,Normal,Normal/images/Normal-9010.png,Normal/masks/Normal-9010.png,True,G18507,train,9970ce038869cbed75784e8c56926ed2,887dd7025e414f1f
Normal-9011.png,Normal,Normal/images/Normal-9011.png,Normal/masks/Normal-9011.png,True,G18508,train,91e771165095a9b3ca6a84b17632c472,897d07965a115f1e
Normal-9012.png,Normal,Normal/images/Normal-9012.png,Normal/masks/Normal-9012.png,True,G18509,val,0afa714e6ffafe14f79519d5f37cba2d,d16e06195d7f1c4c
Normal-9013.png,Normal,Normal/images/Normal-9013.png,Normal/masks/Normal-9013.png,True,G18510,val,c547c019103000217abb5f40dcce4a40,d128521f3a3939bd
Normal-9014.png,Normal,Normal/images/Normal-9014.png,Normal/masks/Normal-9014.png,True,G18511,val,706069f6d716522e964289afdc4509da,d17c421d092eadbd
Normal-9015.png,Normal,Normal/images/Normal-9015.png,Normal/masks/Normal-9015.png,True,G18512,test,08d48e0142d3ac74590a705100ec3606,cc04031b455f7d6f
Normal-9016.png,Normal,Normal/images/Normal-9016.png,Normal/masks/Normal-9016.png,True,G18513,test,3916c2cc32b17c77bc99ecd95bd00353,d97f061a1d0e5f18
Normal-9017.png,Normal,Normal/images/Normal-9017.png,Normal/masks/Normal-9017.png,True,G18514,test,2fd81f77aa98f9e7ef0e008873821f5b,b93d401b154f1d5e
Normal-9018.png,Normal,Normal/images/Normal-9018.png,Normal/masks/Normal-9018.png,True,G18515,train,7eee9074ff6259a1fdd4999cfc993a97,dc3dc2534b424f4b
Normal-9019.png,Normal,Normal/images/Normal-9019.png,Normal/masks/Normal-9019.png,True,G18516,train,2b2aeca50185fb64d9eabb7c2d9ad35d,907b47161e477a5a
Normal-902.png,Normal,Normal/images/Normal-902.png,Normal/masks/Normal-902.png,True,G18517,train,225aa6607577c850ff7a47866bc9d73d,d4fc871bad3d3804
Normal-9020.png,Normal,Normal/images/Normal-9020.png,Normal/masks/Normal-9020.png,True,G18518,train,18e44ee714061118b6a27dcf1b9f07a6,940e474f487c6afc
Normal-9021.png,Normal,Normal/images/Normal-9021.png,Normal/masks/Normal-9021.png,True,G18519,train,66489f8fe71776fbcc3ad15f0dfc1a9a,d92d06c61f4d4f19
Normal-9022.png,Normal,Normal/images/Normal-9022.png,Normal/masks/Normal-9022.png,True,G18520,train,022871c73eb35484e6297f0be93555c4,996a161a4f38e96e
Normal-9023.png,Normal,Normal/images/Normal-9023.png,Normal/masks/Normal-9023.png,True,G18521,train,4f816f3162d50eaa9561a42dd087d802,807f175e59ade0a8
Normal-9024.png,Normal,Normal/images/Normal-9024.png,Normal/masks/Normal-9024.png,True,G18522,val,f5c79256ed7eb62d6a27b0b3f2f75281,992c061b5d7c79cc
Normal-9025.png,Normal,Normal/images/Normal-9025.png,Normal/masks/Normal-9025.png,True,G18523,train,0c9a4bf7c48f83d3322c0a1e8c16e3bc,d17946c21b1de51b
Normal-9026.png,Normal,Normal/images/Normal-9026.png,Normal/masks/Normal-9026.png,True,G18524,train,ab93298a6a94e48a3763b07024851701,dd2a434b0e4f4e2e
Normal-9027.png,Normal,Normal/images/Normal-9027.png,Normal/masks/Normal-9027.png,True,G18525,train,077f533fd89440162a5a6f1920240d56,c02f07435bcc6ed9
Normal-9028.png,Normal,Normal/images/Normal-9028.png,Normal/masks/Normal-9028.png,True,G18526,train,87f57081c33f3efd712b69bc02fd5eb5,d96d061a1f0f1a2f
Normal-9029.png,Normal,Normal/images/Normal-9029.png,Normal/masks/Normal-9029.png,True,G18527,train,e5a42321e3b1dbffb98d4d60aaf92d43,917b461e1b4f3839
Normal-903.png,Normal,Normal/images/Normal-903.png,Normal/masks/Normal-903.png,True,G18528,train,9b0e2b7235b06c4c2f15bdbeb8b3ef2d,d03fc31eb84d3c61
Normal-9030.png,Normal,Normal/images/Normal-9030.png,Normal/masks/Normal-9030.png,True,G18529,train,c10a5dcb04c793895a1bcd3f9d2983bd,dd2c06c61a0bc75f
Normal-9031.png,Normal,Normal/images/Normal-9031.png,Normal/masks/Normal-9031.png,True,G18530,train,8fa19edcf895234faad203c0d4660dd8,937a8d1c1e5e155c
Normal-9032.png,Normal,Normal/images/Normal-9032.png,Normal/masks/Normal-9032.png,True,G18531,test,7c4a0be887427054e788907a7d58f920,b969c61e155f184e
Normal-9033.png,Normal,Normal/images/Normal-9033.png,Normal/masks/Normal-9033.png,True,G18532,val,cd46f7f8dec6a53afcc5a46279769cbe,dd2f46971a4d0d1a
Normal-9034.png,Normal,Normal/images/Normal-9034.png,Normal/masks/Normal-9034.png,True,G18533,val,7271329feb7f5a9eb3b8b9cb11760bc5,9968861a175e5b4f
Normal-9035.png,Normal,Normal/images/Normal-9035.png,Normal/masks/Normal-9035.png,True,G18534,test,c787ce792a3360efd8ee245ab4517ddf,d928c60b1f0e1fbc
Normal-9036.png,Normal,Normal/images/Normal-9036.png,Normal/masks/Normal-9036.png,True,G18535,test,bcd83c17a3f60d4c40a8ee3908bf1492,917f461a192a3d3d
Normal-9037.png,Normal,Normal/images/Normal-9037.png,Normal/masks/Normal-9037.png,True,G18536,train,748802800277f04cfe0989f66601dfe6,c93b079e16851ed5
Normal-9038.png,Normal,Normal/images/Normal-9038.png,Normal/masks/Normal-9038.png,True,G18537,train,e161fb09111c14e4f18354606cbbbc2e,dc2cc3535f420f1d
Normal-9039.png,Normal,Normal/images/Normal-9039.png,Normal/masks/Normal-9039.png,True,G18538,train,5707d106b8716096b8832493ae29c21a,dd7cc34b064b0f43
Normal-904.png,Normal,Normal/images/Normal-904.png,Normal/masks/Normal-904.png,True,G18539,train,0c8f85ee5044565015a6f404dd5bb1d2,d03dc69e3cc035bc
Normal-9040.png,Normal,Normal/images/Normal-9040.png,Normal/masks/Normal-9040.png,True,G18540,train,bb4b4e08137b2bef7df852ca6c92a2b3,913b86191e7e3ba8
Normal-9041.png,Normal,Normal/images/Normal-9041.png,Normal/masks/Normal-9041.png,True,G18541,train,2cee10429fdb1bcd0c8134f8396f57e5,9d2cd0125b435f5b
Normal-9042.png,Normal,Normal/images/Normal-9042.png,Normal/masks/Normal-9042.png,True,G18542,train,d5fbc1ae49b772cb89cbeb838c4420ca,ddbcc61993c61d90
Normal-9043.png,Normal,Normal/images/Normal-9043.png,Normal/masks/Normal-9043.png,True,G18543,train,2fcc487698cd5b326236654a219e8f69,c1b9c65f944a5e58
Normal-9044.png,Normal,Normal/images/Normal-9044.png,Normal/masks/Normal-9044.png,True,G18544,train,678930486b1bb1688124440b45df29d8,807f161e4f1a5a7a
Normal-9045.png,Normal,Normal/images/Normal-9045.png,Normal/masks/Normal-9045.png,True,G18545,train,917e7917c4edb89b4428c5adff900118,997f061a1b29b8e9
Normal-9046.png,Normal,Normal/images/Normal-9046.png,Normal/masks/Normal-9046.png,True,G18546,train,228a299dc3a33b4f2414470909b34d77,d42f935e541d4a0f
Normal-9047.png,Normal,Normal/images/Normal-9047.png,Normal/masks/Normal-9047.png,True,G18547,train,0ec8f7c91e62267f0c308b579e3a0c14,f925061a151b79cf
Normal-9048.png,Normal,Normal/images/Normal-9048.png,Normal/masks/Normal-9048.png,True,G18548,train,52dbbf7295f333308b92f19cc7276a6f,9958961a195e597d
Normal-9049.png,Normal,Normal/images/Normal-9049.png,Normal/masks/Normal-9049.png,True,G18549,train,152010e074036f36829ec4383509ffe8,807d4adf18e9a5b1
Normal-905.png,Normal,Normal/images/Normal-905.png,Normal/masks/Normal-905.png,True,G18550,val,9cccdf58b32147e14410dff98ea781dc,917fae3a294d1c4c
Normal-9050.png,Normal,Normal/images/Normal-9050.png,Normal/masks/Normal-9050.png,True,G18551,val,1488b653bf9a48c4806df59d33cb1ffc,dd2b42171b48ed78
Normal-9051.png,Normal,Normal/images/Normal-9051.png,Normal/masks/Normal-9051.png,True,G18552,val,b0cee0cd3e0958c6230bc2c17c503195,897f061e194f5959
Normal-9052.png,Normal,Normal/images/Normal-9052.png,Normal/masks/Normal-9052.png,True,G18553,test,da188f004bdbc2629f1a4a613e8afe97,dd2ec21b4c4e38f4
Normal-9053.png,Normal,Normal/images/Normal-9053.png,Normal/masks/Normal-9053.png,True,G18554,test,c0f5efd5ea4d027e16f36e2c1a88554e,d92dc61f544f191c
Normal-9054.png,Normal,Normal/images/Normal-9054.png,Normal/masks/Normal-9054.png,True,G18555,test,cb9817ab1e7442e8c2fb37e88baa74d8,d97bd61a5c585858
Normal-9055.png,Normal,Normal/images/Normal-9055.png,Normal/masks/Normal-9055.png,True,G18556,train,d3271698af1b02fa18685650bba21fad,cc3d430d1a2f3db8
Normal-9056.png,Normal,Normal/images/Normal-9056.png,Normal/masks/Normal-9056.png,True,G18557,train,fd0dd2c899565b8fbbba337c1959c55e,896987161ed65e59
Normal-9057.png,Normal,Normal/images/Normal-9057.png,Normal/masks/Normal-9057.png,True,G18558,train,82c2deded29a56d87d001ceaa712274c,817f605fa1867c78
Normal-9058.png,Normal,Normal/images/Normal-9058.png,Normal/masks/Normal-9058.png,True,G18559,train,b090e0f39d0e19ec8b5bcaab10135eba,993b851e59497c78
Normal-9059.png,Normal,Normal/images/Normal-9059.png,Normal/masks/Normal-9059.png,True,G18560,train,9e45f778f84cadd68dc52429db24c2bb,d93f4215170f1a4f
Normal-906.png,Normal,Normal/images/Normal-906.png,Normal/masks/Normal-906.png,True,G18561,train,954f914721b62fe7487550806789fa72,d47ee33fec2c0c04
Normal-9060.png,Normal,Normal/images/Normal-9060.png,Normal/masks/Normal-9060.png,True,G18562,train,69e36c66f81f817f65f89d60994a277c,997887161ac75e5a
Normal-9061.png,Normal,Normal/images/Normal-9061.png,Normal/masks/Normal-9061.png,True,G18563,test,b0d637fa66f7d169ae03b37a1a12757e,99ee071e065a155f
Normal-9062.png,Normal,Normal/images/Normal-9062.png,Normal/masks/Normal-9062.png,True,G18564,train,dc3cf0a126d5a2ae310f45db18d8c1f0,9969c61c155e7a1b
Normal-9063.png,Normal,Normal/images/Normal-9063.png,Normal/masks/Normal-9063.png,True,G18565,val,aeb63b537426fef833c70703c3741eda,997887061b975f19
Normal-9064.png,Normal,Normal/images/Normal-9064.png,Normal/masks/Normal-9064.png,True,G18566,train,d23f3c6d0bef09be27ab848724cd9cbc,c42c03475f4f6c5e
Normal-9065.png,Normal,Normal/images/Normal-9065.png,Normal/masks/Normal-9065.png,True,G18567,val,d32c3bdebac79e0b70e119ab9839090e,b92986161a5e5e7a
Normal-9066.png,Normal,Normal/images/Normal-9066.png,Normal/masks/Normal-9066.png,True,G18568,train,0a7e617bbc4a1946f3a5d813a350fd5f,896c865b5d6d784c
Normal-9067.png,Normal,Normal/images/Normal-9067.png,Normal/masks/Normal-9067.png,True,G18569,test,f399d94e54f37cccd374acfe3dfbc4fd,c4fc8f0b074d5c4d
Normal-9068.png,Normal,Normal/images/Normal-9068.png,Normal/masks/Normal-9068.png,True,G18570,test,bdda834e5422487ab1e28025e3d7e739,dd3cc7831f0d441e
Normal-9069.png,Normal,Normal/images/Normal-9069.png,Normal/masks/Normal-9069.png,True,G18571,train,4a98777000fdb4d82f8486f517a32323,987f041a1b4f5e3a
Normal-907.png,Normal,Normal/images/Normal-907.png,Normal/masks/Normal-907.png,True,G18572,val,5d3181c95e27180331d70a369f78ebc8,91fd863aad3b282c
Normal-9070.png,Normal,Normal/images/Normal-9070.png,Normal/masks/Normal-9070.png,True,G18573,test,2cf45e143cbc7a53da6bb4c0d697f390,89211e5e7a5e6879
Normal-9071.png,Normal,Normal/images/Normal-9071.png,Normal/masks/Normal-9071.png,True,G18574,train,ced25f0992560845b47de81f7608e8ac,d07dc71d1c2e2e2c
Normal-9072.png,Normal,Normal/images/Normal-9072.png,Normal/masks/Normal-9072.png,True,G18575,train,3c0003bcec38d419eb95035c6317e008,dd2fc60f1e4f180c
Normal-9073.png,Normal,Normal/images/Normal-9073.png,Normal/masks/Normal-9073.png,True,G18576,train,232e248be59154cecf3d599c51bb2491,d929061b1d4e5b79
Normal-9074.png,Normal,Normal/images/Normal-9074.png,Normal/masks/Normal-9074.png,True,G18577,train,5a029ba7483bad0fd71274d3a76d2f9d,8178171b5b697879
Normal-9075.png,Normal,Normal/images/Normal-9075.png,Normal/masks/Normal-9075.png,True,G18578,train,916d1c0f939bb377ee601373e9629239,c13c87431a5ee93d
Normal-9076.png,Normal,Normal/images/Normal-9076.png,Normal/masks/Normal-9076.png,True,G18579,train,2a557f3e604ba78b44fd1a5773da85cb,dc3c524b424f5a4f
Normal-9077.png,Normal,Normal/images/Normal-9077.png,Normal/masks/Normal-9077.png,True,G18580,train,ee9e6ab61109fc830b233eb522b18b96,817916161a1dbd7b
Normal-9078.png,Normal,Normal/images/Normal-9078.png,Normal/masks/Normal-9078.png,True,G18581,train,b6d12eda353bc328071534c084f71626,997d065a0d68b9e5
Normal-9079.png,Normal,Normal/images/Normal-9079.png,Normal/masks/Normal-9079.png,True,G18582,train,ed9df79419876a1f8a45913e1cbe4c70,8959961a199f3ba9
Normal-908.png,Normal,Normal/images/Normal-908.png,Normal/masks/Normal-908.png,True,G18583,train,6dd3ab9a208fa892bb5e7ccb5d0d669b,d46ae32b6de0683c
Normal-9080.png,Normal,Normal/images/Normal-9080.png,Normal/masks/Normal-9080.png,True,G18584,train,c1defa8f58f27477e8545788f2d3bcde,952fc31f0a4f1a5a
Normal-9081.png,Normal,Normal/images/Normal-9081.png,Normal/masks/Normal-9081.png,True,G18585,train,bbafc1ba1c545ccfef4d8517bd7e39ae,997b84961a0d5f1b
Normal-9082.png,Normal,Normal/images/Normal-9082.png,Normal/masks/Normal-9082.png,True,G18586,train,d20690c467b726d78c011129a9e8c708,99e9061a4fce38cd
Normal-9083.png,Normal,Normal/images/Normal-9083.png,Normal/masks/Normal-9083.png,True,G18587,train,1f33fd03b7ff1d84e49ae68b77345c53,8c6d42151b4e4f7d
Normal-9084.png,Normal,Normal/images/Normal-9084.png,Normal/masks/Normal-9084.png,True,G18588,train,7dcc74d32cc298626fb1c09bd0786cd4,986f861a4f194b5d
Normal-9085.png,Normal,Normal/images/Normal-9085.png,Normal/masks/Normal-9085.png,True,G18589,train,e255c4c098904f5c2cbe7a419f49e5da,91681e1b1d787bcc
Normal-9086.png,Normal,Normal/images/Normal-9086.png,Normal/masks/Normal-9086.png,True,G18590,val,0e41df16efac937d09d15d24c1f3b33a,d929d61f1c093d69
Normal-9087.png,Normal,Normal/images/Normal-9087.png,Normal/masks/Normal-9087.png,True,G18591,val,99f24b16b4523088984a51d547ec85c9,997fc75e1f041a18
Normal-9088.png,Normal,Normal/images/Normal-9088.png,Normal/masks/Normal-9088.png,True,G18592,val,064ff632585202765a446cdc5a4d93d8,cc3d431d1c2b3de2
Normal-9089.png,Normal,Normal/images/Normal-9089.png,Normal/masks/Normal-9089.png,True,G18593,test,4f479c93b33a4a4a49571532acc3190a,81e9c69a7b874e91
Normal-909.png,Normal,Normal/images/Normal-909.png,Normal/masks/Normal-909.png,True,G18594,test,a0ecb07f053db428fb12b7c696e6a4e4,d16ce71a3b4d3984
Normal-9090.png,Normal,Normal/images/Normal-9090.png,Normal/masks/Normal-9090.png,True,G18595,train,d1007a86b140d314c2808568cfd3d8df,817a46065e5fe8ba
Normal-9091.png,Normal,Normal/images/Normal-9091.png,Normal/masks/Normal-9091.png,True,G18596,test,5a5b35b2ff460c782f8c97f7a3762f39,802b46871b1b5f5f
Normal-9092.png,Normal,Normal/images/Normal-9092.png,Normal/masks/Normal-9092.png,True,G18597,train,f8e86855a702eae2925fa455c098a2e2,dd28c21bd44e1fe4
Normal-9093.png,Normal,Normal/images/Normal-9093.png,Normal/masks/Normal-9093.png,True,G18598,train,349137550e76af922ce227205ef79ce0,d43d435d4a2cbdb0
Normal-9094.png,Normal,Normal/images/Normal-9094.png,Normal/masks/Normal-9094.png,True,G18599,train,56cdb01f0d90a1e65a0bd602ffe5ce8c,997d071e065a394f
Normal-9095.png,Normal,Normal/images/Normal-9095.png,Normal/masks/Normal-9095.png,True,G18600,train,fe7a35f68e0ac9e2c89b1686a22797f7,dc6cc2080cb79aef
Normal-9096.png,Normal,Normal/images/Normal-9096.png,Normal/masks/Normal-9096.png,True,G18601,train,920fa6a58d0b7874ea7ae63cb5f711a5,996806115a4f797f
Normal-9097.png,Normal,Normal/images/Normal-9097.png,Normal/masks/Normal-9097.png,True,G18602,train,0cadca1b822df12da8b1fdebd152dc80,8179d7861b15ce2e
Normal-9098.png,Normal,Normal/images/Normal-9098.png,Normal/masks/Normal-9098.png,True,G18603,train,7810c6458bf76f1ea9ad3bcaafed0fc9,c82fc65a3de8b8a4
Normal-9099.png,Normal,Normal/images/Normal-9099.png,Normal/masks/Normal-9099.png,True,G18604,train,c45fb5e4650dbbe0c51ed9c15f0aa86b,d42ec34b0f647e1c
Normal-91.png,Normal,Normal/images/Normal-91.png,Normal/masks/Normal-91.png,True,G18605,train,750df3465caeca828193cbf3dcf5a94a,917c871a9a3d38ad
Normal-910.png,Normal,Normal/images/Normal-910.png,Normal/masks/Normal-910.png,True,G18606,train,ac43d1a2d18de0f8be5ffdb631826dca,902bca7a78785a9e
Normal-9100.png,Normal,Normal/images/Normal-9100.png,Normal/masks/Normal-9100.png,True,G18607,train,b5b1df3fa7ad7c2a097b8e5bcd87e7fb,d039520b5f0b7b1d
Normal-9101.png,Normal,Normal/images/Normal-9101.png,Normal/masks/Normal-9101.png,True,G18608,train,050d710727659a3576b5f1dfc67a9b4a,d03c4b432c2f2f3d
Normal-9102.png,Normal,Normal/images/Normal-9102.png,Normal/masks/Normal-9102.png,True,G18609,test,e1109a77d47ca99cfa2fd4fdf3d6dcd8,d93bc60b1c4f1b1c
Normal-9103.png,Normal,Normal/images/Normal-9103.png,Normal/masks/Normal-9103.png,True,G18610,test,923ba82e98fbf52730dcc0e37a110b65,dd58521b0b2a2fad
Normal-9104.png,Normal,Normal/images/Normal-9104.png,Normal/masks/Normal-9104.png,True,G18611,train,ead5b29823c792a9fd4c304b7965c7cc,dc2ec35b09461f4d
Normal-9105.png,Normal,Normal/images/Normal-9105.png,Normal/masks/Normal-9105.png,True,G18612,train,cdaad875ac5cd4421e9a72a85398e369,c83dc3155d0e6e3c
Normal-9106.png,Normal,Normal/images/Normal-9106.png,Normal/masks/Normal-9106.png,True,G18613,val,ad2effc88aab880dfb10beaad44fcdde,817b07941e875e97
Normal-9107.png,Normal,Normal/images/Normal-9107.png,Normal/masks/Normal-9107.png,True,G18614,val,1a826e97ab91574311bbb67ef80c09ee,996e86184deeb8e4
Normal-9108.png,Normal,Normal/images/Normal-9108.png,Normal/masks/Normal-9108.png,True,G18615,val,887dc854f7b137574c69c81bdd5456d9,dd2e525f004f1f54
Normal-9109.png,Normal,Normal/images/Normal-9109.png,Normal/masks/Normal-9109.png,True,G18616,test,bd1a975d34495b57ecf4d1517832e524,9928c61b150e3aff
Normal-911.png,Normal,Normal/images/Normal-911.png,Normal/masks/Normal-911.png,True,G18617,train,4777d10f761f1939ef30c2c1f0d87c44,d06ec77be9c468c0
Normal-9110.png,Normal,Normal/images/Normal-9110.png,Normal/masks/Normal-9110.png,True,G18618,train,82504435db3f2b1ca7a07152ca7ee10b,d42c8f43484aecfe
Normal-9111.png,Normal,Normal/images/Normal-9111.png,Normal/masks/Normal-9111.png,True,G18619,train,b4aacf4d7769296c787eef79b56d03dd,9859461e1e495f1f
Normal-9112.png,Normal,Normal/images/Normal-9112.png,Normal/masks/Normal-9112.png,True,G18620,train,bfb9b292a901463c666dd09f6a1c4064,dd3cc24b434a2b2f
Normal-9113.png,Normal,Normal/images/Normal-9113.png,Normal/masks/Normal-9113.png,True,G18621,train,c38818bd6ee208c1d63cac71d2aca37c,cd3cc25b56431e59
Normal-9114.png,Normal,Normal/images/Normal-9114.png,Normal/masks/Normal-9114.png,True,G18622,train,2bcade86a4efbe6c98b27317b0b8a3ea,dd22430f421f5e2f
Normal-9115.png,Normal,Normal/images/Normal-9115.png,Normal/masks/Normal-9115.png,True,G18623,train,a968c4a778925d8dfaea42e15f41c0e2,cc3cc3431b434f3b
Normal-9116.png,Normal,Normal/images/Normal-9116.png,Normal/masks/Normal-9116.png,True,G18624,train,d00a8c355fe0666b79d707f141c5ff98,d939c61e124f1f49
Normal-9117.png,Normal,Normal/images/Normal-9117.png,Normal/masks/Normal-9117.png,True,G18625,train,d4f3c002e64a5ed67889668e23818e37,8968861a9b2fb9ab
Normal-9118.png,Normal,Normal/images/Normal-9118.png,Normal/masks/Normal-9118.png,True,G18626,train,027d5c4d4b91086f807f8b200779e8a4,987cc31b080f7b9d
Normal-9119.png,Normal,Normal/images/Normal-9119.png,Normal/masks/Normal-9119.png,True,G18627,train,549f2f746dc49d1a2ff310f4d981c8f3,886f475e1b4d3a68
Normal-912.png,Normal,Normal/images/Normal-912.png,Normal/masks/Normal-912.png,True,G18628,train,2efd351a694e2985ae3f21f6897b2125,dd30d21f9ce039b8
Normal-9120.png,Normal,Normal/images/Normal-9120.png,Normal/masks/Normal-9120.png,True,G18629,train,2f0eb733cf7d1ed530f6f21c3c96e2b0,d92956821b1d2f7a
Normal-9121.png,Normal,Normal/images/Normal-9121.png,Normal/masks/Normal-9121.png,True,G18630,train,23e9d4a14ba0d20ff8c845757f14a2cd,dc2d420f2cf4b8f8
Normal-9122.png,Normal,Normal/images/Normal-9122.png,Normal/masks/Normal-9122.png,True,G18631,val,c2a602b984f7c744c513efa7deb9a837,dc2cc30b1f082fbd
Normal-9123.png,Normal,Normal/images/Normal-9123.png,Normal/masks/Normal-9123.png,True,G18632,val,cd9e6120c003d98cccc3605bc17feb8f,dd2c424b114f5b4f
Normal-9124.png,Normal,Normal/images/Normal-9124.png,Normal/masks/Normal-9124.png,True,G18633,val,4644918d18fe24b1caf42a902e3a94f7,dd3d42571f42421f
Normal-9125.png,Normal,Normal/images/Normal-9125.png,Normal/masks/Normal-9125.png,True,G18634,test,cd9baeddbc4a30926ed4d2c4613fd284,81f85e15101f5f5b
Normal-9126.png,Normal,Normal/images/Normal-9126.png,Normal/masks/Normal-9126.png,True,G18635,train,7d27c85b52d7ed400c6314d31899eb88,897b07161a5d6b3a
Normal-9127.png,Normal,Normal/images/Normal-9127.png,Normal/masks/Normal-9127.png,True,G18636,test,853f5725cb1f79a6808eeb1623bb731f,817a1619593b39af
Normal-9128.png,Normal,Normal/images/Normal-9128.png,Normal/masks/Normal-9128.png,True,G18637,test,48aa4cf777429617b779ebfa02ae9582,d50a434b5d4f0b1f
Normal-9129.png,Normal,Normal/images/Normal-9129.png,Normal/masks/Normal-9129.png,True,G18638,train,6295e11f89db44d2717b43f6634dc8a3,dd2e470b0f445a5d
Normal-913.png,Normal,Normal/images/Normal-913.png,Normal/masks/Normal-913.png,True,G18639,train,8316b0084aedc46f8fc2ad425e4e6084,913bc61ab8c579b1
Normal-9130.png,Normal,Normal/images/Normal-9130.png,Normal/masks/Normal-9130.png,True,G18640,train,fa60089f2998f088b44291f21d5468a5,c939d79a1c8a5b1c
Normal-9131.png,Normal,Normal/images/Normal-9131.png,Normal/masks/Normal-9131.png,True,G18641,train,f6fdfd3a3e19001f20986104e5a52585,dc2c43432f74b273
Normal-9132.png,Normal,Normal/images/Normal-9132.png,Normal/masks/Normal-9132.png,True,G18642,val,0213638e984c2f351e7752bafb8b1185,d13dc6961b86c61e
Normal-9133.png,Normal,Normal/images/Normal-9133.png,Normal/masks/Normal-9133.png,True,G18643,test,5dfffef6a610837e924c5e5cd8e26bb9,dc3d42431f08ff0d
Normal-9134.png,Normal,Normal/images/Normal-9134.png,Normal/masks/Normal-9134.png,True,G18644,train,0e35d5a395ad3ddeeae7db408a2a5c43,8038575e1b5d69b9
Normal-9135.png,Normal,Normal/images/Normal-9135.png,Normal/masks/Normal-9135.png,True,G18645,train,0852a05dd55136a9212460d83affe6f1,997987561a1c475e
Normal-9136.png,Normal,Normal/images/Normal-9136.png,Normal/masks/Normal-9136.png,True,G18646,train,da92848aa70876a51dbcbc84f5cf05e5,c97d061a1d4979e9
Normal-9137.png,Normal,Normal/images/Normal-9137.png,Normal/masks/Normal-9137.png,True,G18647,test,0fd5ceb1d2748882aa52325c3fc51244,897907161a1fdc7a
Normal-9138.png,Normal,Normal/images/Normal-9138.png,Normal/masks/Normal-9138.png,True,G18648,train,4c8fb3b8ad7461383aee90b2d381208a,dc2a430f560f5c4f
Normal-9139.png,Normal,Normal/images/Normal-9139.png,Normal/masks/Normal-9139.png,True,G18649,train,5c9c49b42ca95e9585758e99fd321bc3,d13d061a596d78e5
Normal-914.png,Normal,Normal/images/Normal-914.png,Normal/masks/Normal-914.png,True,G18650,train,10f4a7786b4577d3c96f17e694976398,9169c71ae94d3cc5
Normal-9140.png,Normal,Normal/images/Normal-9140.png,Normal/masks/Normal-9140.png,True,G18651,train,11a51453f98214fbfde2a9c0ce6e6c29,e979851e18cd5b1a
Normal-9141.png,Normal,Normal/images/Normal-9141.png,Normal/masks/Normal-9141.png,True,G18652,train,e62d4ce7b025fecb011f3470175dac79,816e061a595d7d4f
Normal-9142.png,Normal,Normal/images/Normal-9142.png,Normal/masks/Normal-9142.png,True,G18653,train,101894054862f2bb4878c8bce4ab2014,893a0706164f5e7f
Normal-9143.png,Normal,Normal/images/Normal-9143.png,Normal/masks/Normal-9143.png,True,G18654,val,a8ad1e6c50a1295cbfc6d93be61b6ba9,dd38d2431f484f1d
Normal-9144.png,Normal,Normal/images/Normal-9144.png,Normal/masks/Normal-9144.png,True,G18655,val,a2ace9debf1fda9604557b8f539d96f8,8879859717871f1e
Normal-9145.png,Normal,Normal/images/Normal-9145.png,Normal/masks/Normal-9145.png,True,G18656,test,a704fbe8cf9ed5659fa5d1ffc6c57856,dd2816571b0e393b
Normal-9146.png,Normal,Normal/images/Normal-9146.png,Normal/masks/Normal-9146.png,True,G18657,val,6973d32ce2ac84e69ddadebe18da3b87,985987961a597f1c
Normal-9147.png,Normal,Normal/images/Normal-9147.png,Normal/masks/Normal-9147.png,True,G18658,train,46eb97a8b96b18509788a383fe10ec84,897d06de1a79e838
Normal-9148.png,Normal,Normal/images/Normal-9148.png,Normal/masks/Normal-9148.png,True,G18659,train,aab6b88d9afaba2ff2035ca1b1999be9,c938964f1a4f4f1a
Normal-9149.png,Normal,Normal/images/Normal-9149.png,Normal/masks/Normal-9149.png,True,G18660,test,f2da9c171e8c4c4b18c0ebaa5ccd811d,91699c1e1f5e1873
Normal-915.png,Normal,Normal/images/Normal-915.png,Normal/masks/Normal-915.png,True,G18661,train,565c1da29fbfe563f4e392909ed0800f,c03fe25d38c53d13
Normal-9150.png,Normal,Normal/images/Normal-9150.png,Normal/masks/Normal-9150.png,True,G18662,train,9ccdc223a1057e8bcc19f0ab1199aef2,947c475b114d6f15
Normal-9151.png,Normal,Normal/images/Normal-9151.png,Normal/masks/Normal-9151.png,True,G18663,train,e6c729d88538be57adcaf44214d914b8,9169061f7ba5e0b9
Normal-9152.png,Normal,Normal/images/Normal-9152.png,Normal/masks/Normal-9152.png,True,G18664,train,f1193a190dd7eabf3f7105f691d1668e,813a7e8125e7c63e
Normal-9153.png,Normal,Normal/images/Normal-9153.png,Normal/masks/Normal-9153.png,True,G18665,train,d7d1e8d859fa4501e32a8bc0f5dca92d,d43d4352192e657d
Normal-9154.png,Normal,Normal/images/Normal-9154.png,Normal/masks/Normal-9154.png,True,G18666,train,f67004628e0113b2341b7af364850fd8,f97a965a59181e16
Normal-9155.png,Normal,Normal/images/Normal-9155.png,Normal/masks/Normal-9155.png,True,G18667,train,35a9cbf1e439fb6ea95505f976bb6a92,813b46561bee1f38
Normal-9156.png,Normal,Normal/images/Normal-9156.png,Normal/masks/Normal-9156.png,True,G18668,train,f6de89b69b59620e25ae1035c1e43426,817bd6161e194b5b
Normal-9157.png,Normal,Normal/images/Normal-9157.png,Normal/masks/Normal-9157.png,True,G18669,train,e6e5bc841f9ecbc2243874f19f9a7708,ed48d34b451b4b47
Normal-9158.png,Normal,Normal/images/Normal-9158.png,Normal/masks/Normal-9158.png,True,G18670,train,bd1b3af2ef60e1228c40c2690b8576e6,8979065a7be9e8a1
Normal-9159.png,Normal,Normal/images/Normal-9159.png,Normal/masks/Normal-9159.png,True,G18671,train,0ba843ce51f1883dc30d0f602a8d7eab,d52a535f0f082d1f
Normal-916.png,Normal,Normal/images/Normal-916.png,Normal/masks/Normal-916.png,True,G18672,train,a2043956e2570a505766b2a47253443d,912bc61abc6939e5
Normal-9160.png,Normal,Normal/images/Normal-9160.png,Normal/masks/Normal-9160.png,True,G18673,train,9ffceefcd72c4a17c9bf35ad2afada6c,c96c0e09565c7ced
Normal-9161.png,Normal,Normal/images/Normal-9161.png,Normal/masks/Normal-9161.png,True,G18674,val,1df2dfc7f7166f5f7ddf197ea88e1c28,dd2cd2d30b0bcb83
Normal-9162.png,Normal,Normal/images/Normal-9162.png,Normal/masks/Normal-9162.png,True,G18675,val,085ebc3f9206a7a7c3f7fba216a53dd8,d179d646191d2b39
Normal-9163.png,Normal,Normal/images/Normal-9163.png,Normal/masks/Normal-9163.png,True,G18676,test,739d79a561408f79a72f72d60b97372e,d13c06ce1b4aec79
Normal-9164.png,Normal,Normal/images/Normal-9164.png,Normal/masks/Normal-9164.png,True,G18677,test,d42645a8912d1571607876502598ea24,dd2f421f1d0a2cf8
Normal-9165.png,Normal,Normal/images/Normal-9165.png,Normal/masks/Normal-9165.png,True,G18678,train,92b021b2ffa66101742d3f06428bb39f,c87d421d082d3faf
Normal-9166.png,Normal,Normal/images/Normal-9166.png,Normal/masks/Normal-9166.png,True,G18679,train,49d0d130aa59c103eb166da9479ba355,9968c706da1fec1c
Normal-9167.png,Normal,Normal/images/Normal-9167.png,Normal/masks/Normal-9167.png,True,G18680,train,d95e13d83548d1cb09a6d36f3a914b81,91a866137cc778bc
Normal-9168.png,Normal,Normal/images/Normal-9168.png,Normal/masks/Normal-9168.png,True,G18681,test,753d0281ef6c58e2d4eb7de928638d22,c1395e1716145e5f
Normal-9169.png,Normal,Normal/images/Normal-9169.png,Normal/masks/Normal-9169.png,True,G18682,train,b0ad31de0f34c951ce94ebc8cc062a47,9c0ec21b4dee78e8
Normal-917.png,Normal,Normal/images/Normal-917.png,Normal/masks/Normal-917.png,True,G18683,train,815d54b3be16eb46415858256a88d7c8,9d2c461b0d4d7c4f
Normal-9170.png,Normal,Normal/images/Normal-9170.png,Normal/masks/Normal-9170.png,True,G18684,test,2b42c4fde7c14013b81b648984d2c7ff,817c86521fc71f1e
Normal-9171.png,Normal,Normal/images/Normal-9171.png,Normal/masks/Normal-9171.png,True,G18685,train,30e1fea616beb536c1aa6ebdaa042ae8,d42e42c30b6cedbc
Normal-9172.png,Normal,Normal/images/Normal-9172.png,Normal/masks/Normal-9172.png,True,G18686,train,03cb3cdefc02a20221a7cffe69ff06cc,803f551a584bfbc9
Normal-9173.png,Normal,Normal/images/Normal-9173.png,Normal/masks/Normal-9173.png,True,G18687,train,ff0804d868ed65d5320e95d95d69ca45,9160465e5a5e5f5a
Normal-9174.png,Normal,Normal/images/Normal-9174.png,Normal/masks/Normal-9174.png,True,G18688,train,c2ba091b04921a7a01a47c3e852b1873,dc2d4ad24b0b4f0f
Normal-9175.png,Normal,Normal/images/Normal-9175.png,Normal/masks/Normal-9175.png,True,G18689,train,3bcd57d029848500918f6855ad2ffcaf,d939461a564b1d4f
Normal-9176.png,Normal,Normal/images/Normal-9176.png,Normal/masks/Normal-9176.png,True,G18690,train,9923ba21287ea424c108b60fad9a1283,997f471a13471a5c
Normal-9177.png,Normal,Normal/images/Normal-9177.png,Normal/masks/Normal-9177.png,True,G18691,train,236fbab2bfdd708ccbbd57bd416010bb,dc3d52535e431b4a
Normal-9178.png,Normal,Normal/images/Normal-9178.png,Normal/masks/Normal-9178.png,True,G18692,train,f64074de4ff83746f1de336063ef84ce,d03d4f4a0a6a3d7c
Normal-9179.png,Normal,Normal/images/Normal-9179.png,Normal/masks/Normal-9179.png,True,G18693,val,571494463c98fd8c016da8be5d9dd843,c52d82031fd75f52
Normal-918.png,Normal,Normal/images/Normal-918.png,Normal/masks/Normal-918.png,True,G18694,val,d4029626c2f114ec4909a8115ef9c787,8135c63af96d3991
Normal-9180.png,Normal,Normal/images/Normal-9180.png,Normal/masks/Normal-9180.png,True,G18695,val,534daf9b7150bd2f94bae88f3008f855,c07dc30b0fcf2d2c
Normal-9181.png,Normal,Normal/images/Normal-9181.png,Normal/masks/Normal-9181.png,True,G18696,test,24cdabfa2568d4ad8e134390be989525,d93e865b1c465b4c
Normal-9182.png,Normal,Normal/images/Normal-9182.png,Normal/masks/Normal-9182.png,True,G18697,train,218b858f7e1832d77eeb3a48b35d37d1,817e065a796de929
Normal-9183.png,Normal,Normal/images/Normal-9183.png,Normal/masks/Normal-9183.png,True,G18698,train,fd8e26ffb0cda277b60490c24b0329e9,dd2dc21b495c1ae5
Normal-9184.png,Normal,Normal/images/Normal-9184.png,Normal/masks/Normal-9184.png,True,G18699,train,2ab45e199e99e15751d533ce596f0312,d93a86fa9a057d14
Normal-9185.png,Normal,Normal/images/Normal-9185.png,Normal/masks/Normal-9185.png,True,G18700,train,dfef49268c1336dd71b678fdb27739b8,917b041a5f095f1f
Normal-9186.png,Normal,Normal/images/Normal-9186.png,Normal/masks/Normal-9186.png,True,G18701,train,576f2669d18c14eae1b90b0b327b5572,8979870e1ac57f1a
Normal-9187.png,Normal,Normal/images/Normal-9187.png,Normal/masks/Normal-9187.png,True,G18702,train,678c46882387611670e039939fe8775f,d43c430b2dee3c3c
Normal-9188.png,Normal,Normal/images/Normal-9188.png,Normal/masks/Normal-9188.png,True,G18703,train,21bc2a671f61452f2d181e79f7e2d12d,ccbdd20bc1831fc3
Normal-9189.png,Normal,Normal/images/Normal-9189.png,Normal/masks/Normal-9189.png,True,G18704,train,bc2fb32f07f03eb7b7590d7966e36ae6,99688617175e7ab8
Normal-919.png,Normal,Normal/images/Normal-919.png,Normal/masks/Normal-919.png,True,G18705,train,6bea014939936c85079fab0f6beddad2,99a1c41ec66bb46b
Normal-9190.png,Normal,Normal/images/Normal-9190.png,Normal/masks/Normal-9190.png,True,G18706,train,84cf168e62bfb0a5cba905b7933e1ec6,997f861a491b194f
Normal-9191.png,Normal,Normal/images/Normal-9191.png,Normal/masks/Normal-9191.png,True,G18707,train,9baadcd6321cc17edf73d8ef19ce9920,d03c07421f4f7e0f
Normal-9192.png,Normal,Normal/images/Normal-9192.png,Normal/masks/Normal-9192.png,True,G18708,test,7f21646ebb013ffbbbb5b076ff0274bf,983ec61a555f184f
Normal-9193.png,Normal,Normal/images/Normal-9193.png,Normal/masks/Normal-9193.png,True,G18709,train,79158d41b7739ca4392d6156c05f8166,d97bd61e180b1ba8
Normal-9194.png,Normal,Normal/images/Normal-9194.png,Normal/masks/Normal-9194.png,True,G18710,train,a1bdc83160a8adbfc063adacb4499ae8,992f461a455f584f
Normal-9195.png,Normal,Normal/images/Normal-9195.png,Normal/masks/Normal-9195.png,True,G18711,train,9046fbaa6fec5c8dbd9e7ed9cc40a289,81781e5839ffe0b1
Normal-9196.png,Normal,Normal/images/Normal-9196.png,Normal/masks/Normal-9196.png,True,G18712,val,b4adfec3b2234ac4d208ce46d831d465,cc3d824b1fc44f1d
Normal-9197.png,Normal,Normal/images/Normal-9197.png,Normal/masks/Normal-9197.png,True,G18713,val,3b695648a0a87ae4651cc3e0d28e3752,d97b841782051f5f
Normal-9198.png,Normal,Normal/images/Normal-9198.png,Normal/masks/Normal-9198.png,True,G18714,test,8d1fa162dff2ca76621c5ca062601f11,997d871a055f390d
Normal-9199.png,Normal,Normal/images/Normal-9199.png,Normal/masks/Normal-9199.png,True,G18715,train,1d9b6644390b05974c5c0ba8a4474434,91695e861a5acd7a
Normal-92.png,Normal,Normal/images/Normal-92.png,Normal/masks/Normal-92.png,True,G18716,val,547b9333e667dcc4cc8e0e01ecb33246,d12ad25a5d591d59
Normal-920.png,Normal,Normal/images/Normal-920.png,Normal/masks/Normal-920.png,True,G18717,test,1d498e091c44a8c2b692ad32e610a9c2,c16fc74ace4949c5
Normal-9200.png,Normal,Normal/images/Normal-9200.png,Normal/masks/Normal-9200.png,True,G18718,train,c0aaa9efa4c8d0be5670f9c8f995f046,997a871c09cd3bb1
Normal-9201.png,Normal,Normal/images/Normal-9201.png,Normal/masks/Normal-9201.png,True,G18719,train,75471f4caa082097059b9f9d3e5ff220,890f061e5e5a5d7a
Normal-9202.png,Normal,Normal/images/Normal-9202.png,Normal/masks/Normal-9202.png,True,G18720,train,3c33b5251b147c45f69eaf279d369b54,d92f865a1c445abd
Normal-9203.png,Normal,Normal/images/Normal-9203.png,Normal/masks/Normal-9203.png,True,G18721,train,580fa82b6307d048a0a59d7860a90697,886ec65e1f4d3a07
Normal-9204.png,Normal,Normal/images/Normal-9204.png,Normal/masks/Normal-9204.png,True,G18722,test,2501944c73b9818a679176175fbbb42b,dc0c570b5c4a5f2b
Normal-9205.png,Normal,Normal/images/Normal-9205.png,Normal/masks/Normal-9205.png,True,G18723,train,b91d36089c7d34a02e5bf2649ef2c8e3,d97dc75b45491d01
Normal-9206.png,Normal,Normal/images/Normal-9206.png,Normal/masks/Normal-9206.png,True,G18724,train,142c2375f2cc4c0cede1ff99fc7d0dd0,d95d861a196939e9
Normal-9207.png,Normal,Normal/images/Normal-9207.png,Normal/masks/Normal-9207.png,True,G18725,train,3265602c311352895aa6645fcc0b8c68,992c86195fe8b9c5
Normal-9208.png,Normal,Normal/images/Normal-9208.png,Normal/masks/Normal-9208.png,True,G18726,test,6725e07ba444308e512253ef9c9cd4bb,9dacc21ac51e1d4f
Normal-9209.png,Normal,Normal/images/Normal-9209.png,Normal/masks/Normal-9209.png,True,G18727,train,eae8bde9654fb1c97320cf6d3f7ffcf7,c03f465f184d5f1c
Normal-921.png,Normal,Normal/images/Normal-921.png,Normal/masks/Normal-921.png,True,G18728,train,32fffd1a948b48b7988b6a6ec2eee694,9978c71e16165f38
Normal-9210.png,Normal,Normal/images/Normal-9210.png,Normal/masks/Normal-9210.png,True,G18729,train,eea9930a37ddf1f624505f95378dcef2,d13c4e0f592bec2c
Normal-9211.png,Normal,Normal/images/Normal-9211.png,Normal/masks/Normal-9211.png,True,G18730,train,f545ba06bd77a03160eb4e8bf0633c5d,dc3cd24b0a4b0fad
Normal-9212.png,Normal,Normal/images/Normal-9212.png,Normal/masks/Normal-9212.png,True,G18731,train,45a2a832420d8b7d61ac27a5261ff74d,dc2cd34b4d49491f
Normal-9213.png,Normal,Normal/images/Normal-9213.png,Normal/masks/Normal-9213.png,True,G18732,train,2fa31e695ae6b769f51e017a50b9e8cd,d97dc71a954a1949
Normal-9214.png,Normal,Normal/images/Normal-9214.png,Normal/masks/Normal-9214.png,True,G18733,train,bf1aa3a67942b95e51016b00f52fa6c9,c13b565e181c4f3b
Normal-9215.png,Normal,Normal/images/Normal-9215.png,Normal/masks/Normal-9215.png,True,G18734,val,65e00832be44d8a915feff2e015c5f4e,8179861a19ed7a79
Normal-9216.png,Normal,Normal/images/Normal-9216.png,Normal/masks/Normal-9216.png,True,G18735,val,359be2127de572aafb14c6d1c3ced038,c835821f1e4f3d2d
Normal-9217.png,Normal,Normal/images/Normal-9217.png,Normal/masks/Normal-9217.png,True,G18736,val,be97ace5da5fa3fdac73369dc487a3d8,c43d4b0b0d2c3dbe
Normal-9218.png,Normal,Normal/images/Normal-9218.png,Normal/masks/Normal-9218.png,True,G18737,test,183f96b76162c31b9be720ab7cc8f1f3,db69851bc50a1d4f
Normal-9219.png,Normal,Normal/images/Normal-9219.png,Normal/masks/Normal-9219.png,True,G18738,train,21d33ef548ed971c3c67970287b88231,817d85861a1bcd5f
Normal-922.png,Normal,Normal/images/Normal-922.png,Normal/masks/Normal-922.png,True,G18739,train,b2463bc815c43d40c8cc569fb698437b,94edc71acb292c6c
Normal-9220.png,Normal,Normal/images/Normal-9220.png,Normal/masks/Normal-9220.png,True,G18740,train,d720e8b5572703b59760ee6ea2433bc8,dc2c021bda6e38fc
Normal-9221.png,Normal,Normal/images/Normal-9221.png,Normal/masks/Normal-9221.png,True,G18741,train,a5673939de1efe8d675d2893068dfa9d,893b041e5f485b7d
Normal-9222.png,Normal,Normal/images/Normal-9222.png,Normal/masks/Normal-9222.png,True,G18742,train,8ee8dd2acd27ba7719bef10ec2a85993,987b85151e5e1f1a
Normal-9223.png,Normal,Normal/images/Normal-9223.png,Normal/masks/Normal-9223.png,True,G18743,train,28e7d721339725c3bd43059179fbc957,dd2ec61b4d425bc4
Normal-9224.png,Normal,Normal/images/Normal-9224.png,Normal/masks/Normal-9224.png,True,G18744,train,c859447708922565a7cc925768bbbefa,d43d83091d4f3d9a
Normal-9225.png,Normal,Normal/images/Normal-9225.png,Normal/masks/Normal-9225.png,True,G18745,train,4c96f3f4b32a5383252618d23c01eacf,fd2ad30b57423d42
Normal-9226.png,Normal,Normal/images/Normal-9226.png,Normal/masks/Normal-9226.png,True,G18746,train,df2039ba757aa80c6461c25084fd3a20,802bc5c71e96d64f
Normal-9227.png,Normal,Normal/images/Normal-9227.png,Normal/masks/Normal-9227.png,True,G18747,train,a81d0e2ac1f6e759b0dd3b35b8262cf1,983fd71a405b595a
Normal-9228.png,Normal,Normal/images/Normal-9228.png,Normal/masks/Normal-9228.png,True,G18748,train,455927baada5caabca70409216ba12f9,99681659986e79e5
Normal-9229.png,Normal,Normal/images/Normal-9229.png,Normal/masks/Normal-9229.png,True,G18749,train,c2ffec996a8f73e88f84d1690d6b5e8d,d92dd20b1f461b1d
Normal-923.png,Normal,Normal/images/Normal-923.png,Normal/masks/Normal-923.png,True,G18750,test,72ee465fd3457db94bbfb48dd9f9e15c,d47dc71bad0d18c4
Normal-9230.png,Normal,Normal/images/Normal-9230.png,Normal/masks/Normal-9230.png,True,G18751,test,d73d8e3be04c8ea633914d691ba489e8,bc7d821bc3161ac7
Normal-9231.png,Normal,Normal/images/Normal-9231.png,Normal/masks/Normal-9231.png,True,G18752,train,e30de1627194e64773b8c23b2dde64d4,8929c61e5c5f1a5d
Normal-9232.png,Normal,Normal/images/Normal-9232.png,Normal/masks/Normal-9232.png,True,G18753,train,a6498d67c4e98c0c3c2a66f8b283b60d,f680e9e786c44e5c
Normal-9233.png,Normal,Normal/images/Normal-9233.png,Normal/masks/Normal-9233.png,True,G18754,train,6a0ebcbf7c91557744a13dcca0ab4aa8,8c2d94c31f0f4f0f
Normal-9234.png,Normal,Normal/images/Normal-9234.png,Normal/masks/Normal-9234.png,True,G18755,train,906cfea35434326bb922dd082982fa00,d03c4f43090bef3d
Normal-9235.png,Normal,Normal/images/Normal-9235.png,Normal/masks/Normal-9235.png,True,G18756,val,a82ce157cfb44c946b64dee0482a21bd,81394e1f5a2f194f
Normal-9236.png,Normal,Normal/images/Normal-9236.png,Normal/masks/Normal-9236.png,True,G18757,val,cdbf5b3472f1406c659a1ffc31c68c04,9978061c175e397b
Normal-9237.png,Normal,Normal/images/Normal-9237.png,Normal/masks/Normal-9237.png,True,G18758,val,7a14df23e3abc725e09c64aa8003a823,81f946071e174f1f
Normal-9238.png,Normal,Normal/images/Normal-9238.png,Normal/masks/Normal-9238.png,True,G18759,test,4ebd5a379d5447bdd35045b4afccf6f6,c87dc30f460f1a4f
Normal-9239.png,Normal,Normal/images/Normal-9239.png,Normal/masks/Normal-9239.png,True,G18760,train,20b31f6944c591a3131ed54b52836091,993d861a1dc41bbd
Normal-924.png,Normal,Normal/images/Normal-924.png,Normal/masks/Normal-924.png,True,G18761,train,1d2b5f86637c8102b1def8f315bd6967,c13d6f1938c33e92
Normal-9240.png,Normal,Normal/images/Normal-9240.png,Normal/masks/Normal-9240.png,True,G18762,train,a826439c889ab4c8767e2583f1d102cb,d85d421a283b6d7d
Normal-9241.png,Normal,Normal/images/Normal-9241.png,Normal/masks/Normal-9241.png,True,G18763,train,8db8b1831fd3919acad8887e6f11e6a6,d92d061ec47ae86b
Normal-9242.png,Normal,Normal/images/Normal-9242.png,Normal/masks/Normal-9242.png,True,G18764,train,857c7e51667a300bbc700caa4eedd9e7,d07b535f4a083a1f
Normal-9243.png,Normal,Normal/images/Normal-9243.png,Normal/masks/Normal-9243.png,True,G18765,train,4166114bb189892c38b3b5d8e558453a,dd2ad30fca492f28
Normal-9244.png,Normal,Normal/images/Normal-9244.png,Normal/masks/Normal-9244.png,True,G18766,train,5df98aea6dfba747b2d5d0a6eca91cee,dd2ec25f024f1f14
Normal-9245.png,Normal,Normal/images/Normal-9245.png,Normal/masks/Normal-9245.png,True,G18767,train,b4a240015b0820dc3973922831ef5bc5,9979871a1b4d1a79
Normal-9246.png,Normal,Normal/images/Normal-9246.png,Normal/masks/Normal-9246.png,True,G18768,train,f293d68b5820760b93901175912b2f51,9979c5865a0ec753
Normal-9247.png,Normal,Normal/images/Normal-9247.png,Normal/masks/Normal-9247.png,True,G18769,train,e18dcf258b6321565ee2dc9c22023e6c,897d861f1d4d1b19
Normal-9248.png,Normal,Normal/images/Normal-9248.png,Normal/masks/Normal-9248.png,True,G18770,train,33ffe29acd9240fa8bd49b82f9820147,c43d434b0c2ea3fd
Normal-9249.png,Normal,Normal/images/Normal-9249.png,Normal/masks/Normal-9249.png,True,G18771,test,d5235f10c828162101ccc42e3a86a537,d90cc743196d6d3c
Normal-925.png,Normal,Normal/images/Normal-925.png,Normal/masks/Normal-925.png,True,G18772,val,0670bc3f5bb7f4f28232b47f401b27c3,d03bc71abcf0389c
Normal-9250.png,Normal,Normal/images/Normal-9250.png,Normal/masks/Normal-9250.png,True,G18773,test,53ce204d9bc46249e5a8e397cde3ea56,c979c60615cd5b1b
Normal-9251.png,Normal,Normal/images/Normal-9251.png,Normal/masks/Normal-9251.png,True,G18774,val,8b811c4e5966ba7cda16c81bc486fbcc,997bc71a155e3845
Normal-9252.png,Normal,Normal/images/Normal-9252.png,Normal/masks/Normal-9252.png,True,G18775,test,4067dcb4d562ee4485e88a659c3cc8ae,d0784a1b594f3d39
Normal-9253.png,Normal,Normal/images/Normal-9253.png,Normal/masks/Normal-9253.png,True,G18776,val,60b46d8d7019cc66b5621c82797bdcf7,cd2cd14b1f470b1d
Normal-9254.png,Normal,Normal/images/Normal-9254.png,Normal/masks/Normal-9254.png,True,G18777,train,0af58c72fa71b625be77bf0ed2a03a5c,dd2a871e194d5a59
Normal-9255.png,Normal,Normal/images/Normal-9255.png,Normal/masks/Normal-9255.png,True,G18778,train,19ce944188d77bd07b76896dd1dd81ef,997ac7121b075a1f
Normal-9256.png,Normal,Normal/images/Normal-9256.png,Normal/masks/Normal-9256.png,True,G18779,train,ace363cbd553bb65872b5802f6e4e43b,dc2ec39309451f1f
Normal-9257.png,Normal,Normal/images/Normal-9257.png,Normal/masks/Normal-9257.png,True,G18780,train,cc33b15025973a723cc387e19a80be06,dd2dc2931bc4869b
Normal-9258.png,Normal,Normal/images/Normal-9258.png,Normal/masks/Normal-9258.png,True,G18781,train,5c75b3f7f2584dcba22a7c4e19ecb913,d81d420f1c2d3dbd
Normal-9259.png,Normal,Normal/images/Normal-9259.png,Normal/masks/Normal-9259.png,True,G18782,train,2fc3e82f548a73dcb3762d599c9aabd4,9979871a5d0d1a9d
Normal-926.png,Normal,Normal/images/Normal-926.png,Normal/masks/Normal-926.png,True,G18783,train,b45f6e4403975781014f7ee2e1b5acdf,8c3fc71f4d5058d2
Normal-9260.png,Normal,Normal/images/Normal-9260.png,Normal/masks/Normal-9260.png,True,G18784,train,bc49fa5806eda0bbb1991954ddfd9876,dd6d861ac71e114e
Normal-9261.png,Normal,Normal/images/Normal-9261.png,Normal/masks/Normal-9261.png,True,G18785,train,e791b704c34d8b36ba263f66b5df573d,d53c46c79b1b884b
Normal-9262.png,Normal,Normal/images/Normal-9262.png,Normal/masks/Normal-9262.png,True,G18786,train,361453298b0eaf6232f2ec24766d9aaf,c93dc21f124e3f19
Normal-9263.png,Normal,Normal/images/Normal-9263.png,Normal/masks/Normal-9263.png,True,G18787,train,41d01b3017ee92760956b402b75a76af,bd38c30b1f550a5d
Normal-9264.png,Normal,Normal/images/Normal-9264.png,Normal/masks/Normal-9264.png,True,G18788,train,9c8411274b5849f6580706a88cd94938,d979071e582939e9
Normal-9265.png,Normal,Normal/images/Normal-9265.png,Normal/masks/Normal-9265.png,True,G18789,train,cafc09316356a093376142cd5b1ff4cd,997b061a18557bb9
Normal-9266.png,Normal,Normal/images/Normal-9266.png,Normal/masks/Normal-9266.png,True,G18790,val,29f8c521a6b807df0147a30d76db527a,c83f56069f14eda1
Normal-9267.png,Normal,Normal/images/Normal-9267.png,Normal/masks/Normal-9267.png,True,G18791,train,93701cd042143586712e4b92e978ece1,dd3c425b0bec2db0
Normal-9268.png,Normal,Normal/images/Normal-9268.png,Normal/masks/Normal-9268.png,True,G18792,train,df24cf2afebbbe50c491e32b3d202332,cc7c421b474f386d
Normal-9269.png,Normal,Normal/images/Normal-9269.png,Normal/masks/Normal-9269.png,True,G18793,val,7deb5139abb8cf62282ea822004964c3,8178861a5f4f3a79
Normal-927.png,Normal,Normal/images/Normal-927.png,Normal/masks/Normal-927.png,True,G18794,test,eaa4b690a7798a128cfe2ae28e4ead24,916dc61acd49b9c3
Normal-9270.png,Normal,Normal/images/Normal-9270.png,Normal/masks/Normal-9270.png,True,G18795,test,77a0346960bdb3a9e76cec9bc28ca90e,9169061e1b5f5b59
Normal-9271.png,Normal,Normal/images/Normal-9271.png,Normal/masks/Normal-9271.png,True,G18796,test,2222ad9e1a564eaccd875d5e99ab9188,d96b561a1949193f
Normal-9272.png,Normal,Normal/images/Normal-9272.png,Normal/masks/Normal-9272.png,True,G18797,train,d9b469599789a08c76f8609e85c9ef2c,997f061849e1bae6
Normal-9273.png,Normal,Normal/images/Normal-9273.png,Normal/masks/Normal-9273.png,True,G18798,val,100d7a29471d4008cad8525fce01ea5f,dd2cd35f444a1d4c
Normal-9274.png,Normal,Normal/images/Normal-9274.png,Normal/masks/Normal-9274.png,True,G18799,train,e79c5423c6a60fc67e6f66166dc5f6c9,9c3e454e5b4d5c4c
Normal-9275.png,Normal,Normal/images/Normal-9275.png,Normal/masks/Normal-9275.png,True,G18800,train,d8d1d7080b8437b367e0743d80ff2991,9d5dd61e4c581a0f
Normal-9276.png,Normal,Normal/images/Normal-9276.png,Normal/masks/Normal-9276.png,True,G18801,train,afdf85dcc7b7d62ddf2a6f3189c7be85,dc2fc30b0cac3cf4
Normal-9277.png,Normal,Normal/images/Normal-9277.png,Normal/masks/Normal-9277.png,True,G18802,train,a1a80c834811f86b64539bc537af237d,d83dd25713c24f1c
Normal-9278.png,Normal,Normal/images/Normal-9278.png,Normal/masks/Normal-9278.png,True,G18803,train,c47b6b2215399ee1d3074ee8fca370af,dd3c4e184cccb8cd
Normal-9279.png,Normal,Normal/images/Normal-9279.png,Normal/masks/Normal-9279.png,True,G18804,train,e15317a489b56b519c496ad7e4ebe929,c0b447ca7acb3dc2
Normal-928.png,Normal,Normal/images/Normal-928.png,Normal/masks/Normal-928.png,True,G18805,train,a9da5844df2788f75e259e694e884291,9123863ae97c6975
Normal-9280.png,Normal,Normal/images/Normal-9280.png,Normal/masks/Normal-9280.png,True,G18806,train,c3dfe6829ddbda1647245daeaf984e9b,982d069a07de3dcd
Normal-9281.png,Normal,Normal/images/Normal-9281.png,Normal/masks/Normal-9281.png,True,G18807,train,83afdeb6f0ccd1b8ea0648ab1fe583b7,9979861a4d1b394f
Normal-9282.png,Normal,Normal/images/Normal-9282.png,Normal/masks/Normal-9282.png,True,G18808,train,6b5dcb172ca65977f3b5968e45811afd,d42f4b5f0d0a2d3c
Normal-9283.png,Normal,Normal/images/Normal-9283.png,Normal/masks/Normal-9283.png,True,G18809,train,f44db8cbe88796a034b4fa07f6507e32,997985165a4c4d7b
Normal-9284.png,Normal,Normal/images/Normal-9284.png,Normal/masks/Normal-9284.png,True,G18810,val,fa29a0f1c01ec163d4f801ecf92b7d9b,886d871f1f0d1e4d
Normal-9285.png,Normal,Normal/images/Normal-9285.png,Normal/masks/Normal-9285.png,True,G18811,train,c32c1e4a0033d60a66848edb807a0fed,993d04521b4d4f3f
Normal-9286.png,Normal,Normal/images/Normal-9286.png,Normal/masks/Normal-9286.png,True,G18812,train,98a199cc5c1e8ae5c4c77910c700c82f,dd39c64e78c24e38
Normal-9287.png,Normal,Normal/images/Normal-9287.png,Normal/masks/Normal-9287.png,True,G18813,val,600add205b85b9660e6c62e0c05d102a,9939c61e114f7ab8
Normal-9288.png,Normal,Normal/images/Normal-9288.png,Normal/masks/Normal-9288.png,True,G18814,val,88f032df395ca6baaa0f75042535125f,d1ff86527944861b
Normal-9289.png,Normal,Normal/images/Normal-9289.png,Normal/masks/Normal-9289.png,True,G18815,test,f24880dabfe4fb8a452679675d97c1db,c979c61e184f1f19
Normal-929.png,Normal,Normal/images/Normal-929.png,Normal/masks/Normal-929.png,True,G18816,test,e872601bacc99839afc8e005d0411154,c17ce43c2d4c2ece
Normal-9290.png,Normal,Normal/images/Normal-9290.png,Normal/masks/Normal-9290.png,True,G18817,test,28eb1ea17343e7c087437b3054536a28,8169161e1e5f1b5e
Normal-9291.png,Normal,Normal/images/Normal-9291.png,Normal/masks/Normal-9291.png,True,G18818,train,40b7afaddc5d7780fcc038d8157bdff8,dd2cd24b1f0c6939
Normal-9292.png,Normal,Normal/images/Normal-9292.png,Normal/masks/Normal-9292.png,True,G18819,train,a81ab8bca10fc982abaa54a1c5d795c4,d97dc71fc44a184a
Normal-9293.png,Normal,Normal/images/Normal-9293.png,Normal/masks/Normal-9293.png,True,G18820,train,ed9ce59081b782e33c17563ee562b912,91681c1e1b5e7b78
Normal-9294.png,Normal,Normal/images/Normal-9294.png,Normal/masks/Normal-9294.png,True,G18821,train,9b633a02e477ee8617d976808644bb51,cd5cc21f1c6c2d39
Normal-9295.png,Normal,Normal/images/Normal-9295.png,Normal/masks/Normal-9295.png,True,G18822,train,6a6f2d320cf229888fe47f8742c2a53a,dd2ed353450a1d4d
Normal-9296.png,Normal,Normal/images/Normal-9296.png,Normal/masks/Normal-9296.png,True,G18823,train,b54a05a97f8b9b35a716efd1c8840a60,d83dd30b0d181dbd
Normal-9297.png,Normal,Normal/images/Normal-9297.png,Normal/masks/Normal-9297.png,True,G18824,train,5c98619d2d4ec614a28774eb56c297fd,892fc61a144d7f71
Normal-9298.png,Normal,Normal/images/Normal-9298.png,Normal/masks/Normal-9298.png,True,G18825,train,46c9f63f7edaa4b76b564e732ee86893,917f9618597aa9a2
Normal-9299.png,Normal,Normal/images/Normal-9299.png,Normal/masks/Normal-9299.png,True,G18826,train,1ca472d7baf725965722778a7494e9bd,91798646115fcf3a
Normal-93.png,Normal,Normal/images/Normal-93.png,Normal/masks/Normal-93.png,True,G18827,train,024519648c067c5ec38535413c4787c5,887f071e0e1f384f
Normal-930.png,Normal,Normal/images/Normal-930.png,Normal/masks/Normal-930.png,True,G18828,train,7bcea9cf73a10d2c794f017982711ab5,d46dc70bcd4c1cc3
Normal-9300.png,Normal,Normal/images/Normal-9300.png,Normal/masks/Normal-9300.png,True,G18829,val,96239cbf7d105ff37d214df128e541da,fd7dc2125b490f18
Normal-9301.png,Normal,Normal/images/Normal-9301.png,Normal/masks/Normal-9301.png,True,G18830,train,4af2ddc3a858bc888a61411f115b9852,997f561e544b184b
Normal-9302.png,Normal,Normal/images/Normal-9302.png,Normal/masks/Normal-9302.png,True,G18831,train,656232bbdedb598bfbc9a28ea867e092,c07d474b0f0e2f0f
Normal-9303.png,Normal,Normal/images/Normal-9303.png,Normal/masks/Normal-9303.png,True,G18832,val,7146498c12a77505a8d76390155ea38e,dc2fd35b0f440c2d
Normal-9304.png,Normal,Normal/images/Normal-9304.png,Normal/masks/Normal-9304.png,True,G18833,val,aa06766ffd0dca79e600f39646e7033f,997b061c19e9e1b6
Normal-9305.png,Normal,Normal/images/Normal-9305.png,Normal/masks/Normal-9305.png,True,G18834,test,1ef28cc64b218c5dfea9dcd706c391bf,dd2cc21f460f1c4f
Normal-9306.png,Normal,Normal/images/Normal-9306.png,Normal/masks/Normal-9306.png,True,G18835,train,caac964b90ed5496b69d5997d238e178,fd2ec30f444e3cc4
Normal-9307.png,Normal,Normal/images/Normal-9307.png,Normal/masks/Normal-9307.png,True,G18836,test,ccc3cf5d83b3e2a5b666cf7763ae4cf7,d93cc61b144f39e1
Normal-9308.png,Normal,Normal/images/Normal-9308.png,Normal/masks/Normal-9308.png,True,G18837,test,d3ebfcfe9d9c242fb15b710691e4fdf1,913fc61b15451f1d
Normal-9309.png,Normal,Normal/images/Normal-9309.png,Normal/masks/Normal-9309.png,True,G18838,train,e235247e753d1811f3afe56843086e24,d969c61a554b7949
Normal-931.png,Normal,Normal/images/Normal-931.png,Normal/masks/Normal-931.png,True,G18839,train,aa1f8511a99313135d405b1c3d4a3c77,d46ec70fcd0d68c4
Normal-9310.png,Normal,Normal/images/Normal-9310.png,Normal/masks/Normal-9310.png,True,G18840,train,6b3863a3fbd19eab93b420564368a344,9168c65e584d7c79
Normal-9311.png,Normal,Normal/images/Normal-9311.png,Normal/masks/Normal-9311.png,True,G18841,train,4e1708c90835b27aa3c775d53268df93,d12d074b1fd22db4
Normal-9312.png,Normal,Normal/images/Normal-9312.png,Normal/masks/Normal-9312.png,True,G18842,train,892ecfd4c684d7158b73565ad6bdc045,dc2e43530b6d0f3c
Normal-9313.png,Normal,Normal/images/Normal-9313.png,Normal/masks/Normal-9313.png,True,G18843,train,ee41480ba7fb9dfd00f4327a52b900ab,853c46431bbcec3d
Normal-9314.png,Normal,Normal/images/Normal-9314.png,Normal/masks/Normal-9314.png,True,G18844,test,1efc248b576cb107a4d90baa87919eed,9938861e194f1efc
Normal-9315.png,Normal,Normal/images/Normal-9315.png,Normal/masks/Normal-9315.png,True,G18845,train,0290642da21c4362aed8dea96c0cde35,d47e42531b4b1f0d
Normal-9316.png,Normal,Normal/images/Normal-9316.png,Normal/masks/Normal-9316.png,True,G18846,train,b0265653196efb37a5e0abfbfcc759bc,9959c61b19457f38
Normal-9317.png,Normal,Normal/images/Normal-9317.png,Normal/masks/Normal-9317.png,True,G18847,train,38b7d0dedfde9dca9b7b40904ee273b7,813916565b7ee878
Normal-9318.png,Normal,Normal/images/Normal-9318.png,Normal/masks/Normal-9318.png,True,G18848,train,ad2b2f5c5c57c510d25deba56c1dd077,d939c61a186f3d2c
Normal-9319.png,Normal,Normal/images/Normal-9319.png,Normal/masks/Normal-9319.png,True,G18849,train,5a95448fbd664643fd8cc090197996e5,89b907927ac778e1
Normal-932.png,Normal,Normal/images/Normal-932.png,Normal/masks/Normal-932.png,True,G18850,train,3c21b3641f5c1c8c8948d400f882f9c6,91fcc71b9dc418d2
Normal-9320.png,Normal,Normal/images/Normal-9320.png,Normal/masks/Normal-9320.png,True,G18851,train,d9545242c191d99bb74bec54f4e50313,917a06197beca9b8
Normal-9321.png,Normal,Normal/images/Normal-9321.png,Normal/masks/Normal-9321.png,True,G18852,val,92d5c86faa2e9b97a05db7db65fd750e,9978871a598f1ae9
Normal-9322.png,Normal,Normal/images/Normal-9322.png,Normal/masks/Normal-9322.png,True,G18853,val,8a178410b28034ecd6f361cb7b931fef,9939c717984f7950
Normal-9323.png,Normal,Normal/images/Normal-9323.png,Normal/masks/Normal-9323.png,True,G18854,val,29a19857ab2217a9149844125531f766,dc2e42c74bc22f9a
Normal-9324.png,Normal,Normal/images/Normal-9324.png,Normal/masks/Normal-9324.png,True,G18855,train,19947e74dea9c2370e53bca08105a98b,913a02595d7a7be2
Normal-9325.png,Normal,Normal/images/Normal-9325.png,Normal/masks/Normal-9325.png,True,G18856,train,9fe66c32276690eeef7d6025c316810f,dd680e5e195f2919
Normal-9326.png,Normal,Normal/images/Normal-9326.png,Normal/masks/Normal-9326.png,True,G18857,train,e74aeedfa7c6fff26dd07dff5f52a79d,d43dc30b0a4f3d4e
Normal-9327.png,Normal,Normal/images/Normal-9327.png,Normal/masks/Normal-9327.png,True,G18858,test,89631be7a9ec092200053b99670519a4,dd2de0121f4b4b1d
Normal-9328.png,Normal,Normal/images/Normal-9328.png,Normal/masks/Normal-9328.png,True,G18859,test,3a2d7004fc1a2a5f0069077def35d872,d91907161a59797b
Normal-9329.png,Normal,Normal/images/Normal-9329.png,Normal/masks/Normal-9329.png,True,G18860,train,b4025c79c49917612cc2dcd6b5b9856d,dc2d43570a564f1e
Normal-933.png,Normal,Normal/images/Normal-933.png,Normal/masks/Normal-933.png,True,G18861,val,d4b9b0e5f71ad0e387d966e167c0c417,903bd61e1d159597
Normal-9330.png,Normal,Normal/images/Normal-9330.png,Normal/masks/Normal-9330.png,True,G18862,train,6ac02d56d20205840690e0ca60bc7422,8139c6161a5f7a79
Normal-9331.png,Normal,Normal/images/Normal-9331.png,Normal/masks/Normal-9331.png,True,G18863,train,c9165a126b61d55aaefef1c74235632d,83708c8d344b8f7f
Normal-9332.png,Normal,Normal/images/Normal-9332.png,Normal/masks/Normal-9332.png,True,G18864,train,50d4048b11315a056094c2c6a5c59142,d92c821b1fce2ab9
Normal-9333.png,Normal,Normal/images/Normal-9333.png,Normal/masks/Normal-9333.png,True,G18865,train,72c032030b10f82340eb56a780aaf1ef,dd2d401f1b455f19
Normal-9334.png,Normal,Normal/images/Normal-9334.png,Normal/masks/Normal-9334.png,True,G18866,train,d2836420228f4e8768e4b284107d4779,d13b461919adb8bc
Normal-9335.png,Normal,Normal/images/Normal-9335.png,Normal/masks/Normal-9335.png,True,G18867,val,1e10f557e10b124911bd8f5d32bd4a29,d42d02131f0eee7d
Normal-9336.png,Normal,Normal/images/Normal-9336.png,Normal/masks/Normal-9336.png,True,G18868,train,21e93f4b6a1e00c9727558a8a37d3736,d93946461e9c6a7a
Normal-9337.png,Normal,Normal/images/Normal-9337.png,Normal/masks/Normal-9337.png,True,G18869,train,f4c475a96bd6466d67793f91e1d48966,dd2cd2431b430f5b
Normal-9338.png,Normal,Normal/images/Normal-9338.png,Normal/masks/Normal-9338.png,True,G18870,test,b1c16c0d127b402669f43e5d15cb1571,c97d524b555a5d44
Normal-9339.png,Normal,Normal/images/Normal-9339.png,Normal/masks/Normal-9339.png,True,G18871,train,3860d4328cb61a0593de7aec16c64c52,dc3dc21d4a0f1d4b
Normal-934.png,Normal,Normal/images/Normal-934.png,Normal/masks/Normal-934.png,True,G18872,train,84bab56eddfbf84472f6163ec85a8c79,810ae71bf8c54bd3
Normal-9340.png,Normal,Normal/images/Normal-9340.png,Normal/masks/Normal-9340.png,True,G18873,train,780113f3884d10fc899b59608596953b,802857c71f577e19
Normal-9341.png,Normal,Normal/images/Normal-9341.png,Normal/masks/Normal-9341.png,True,G18874,val,0feea335ea396e6599edaaed95fb8f9f,81687e075a196f67
Normal-9342.png,Normal,Normal/images/Normal-9342.png,Normal/masks/Normal-9342.png,True,G18875,val,8def5d147c6993b6c5f2ca92c7de7ef5,c03b57161e1f4d1b
Normal-9343.png,Normal,Normal/images/Normal-9343.png,Normal/masks/Normal-9343.png,True,G18876,test,7001539d9054de27faa17b4a6ec28c60,d16946945ac67ada
Normal-9344.png,Normal,Normal/images/Normal-9344.png,Normal/masks/Normal-9344.png,True,G18877,test,c220dc59b13573a3765028f17ebfb04b,886fc5161e5f0b5c
Normal-9345.png,Normal,Normal/images/Normal-9345.png,Normal/masks/Normal-9345.png,True,G18878,train,001d1562ea33227be0f24ec4eebb99a3,d969c61ad54e18c7
Normal-9346.png,Normal,Normal/images/Normal-9346.png,Normal/masks/Normal-9346.png,True,G18879,train,fc47b7ebf8c77e7037e660f01e805be7,817ad65a58477b91
Normal-9347.png,Normal,Normal/images/Normal-9347.png,Normal/masks/Normal-9347.png,True,G18880,train,de5fa17dde840b229a295a61cdd91c67,9d3cc2c257474717
Normal-9348.png,Normal,Normal/images/Normal-9348.png,Normal/masks/Normal-9348.png,True,G18881,train,4b82c6880d7a84d18ab4cc159ec801a7,d42d02d35b14dbcb
Normal-9349.png,Normal,Normal/images/Normal-9349.png,Normal/masks/Normal-9349.png,True,G18882,train,f72c98df79c96ec4096cefb70cd9d4a9,9179c7861a1ac55f
Normal-935.png,Normal,Normal/images/Normal-935.png,Normal/masks/Normal-935.png,True,G18883,train,363b00660060fe9137863a4f1ef4118b,917d8e1a89589ccf
Normal-9350.png,Normal,Normal/images/Normal-9350.png,Normal/masks/Normal-9350.png,True,G18884,train,994e7a7910e01b48e0a55a9eb3fc4082,81285e1f7e4e78c5
Normal-9351.png,Normal,Normal/images/Normal-9351.png,Normal/masks/Normal-9351.png,True,G18885,train,404db5c3085302e398e48f29dd3f20b3,d42e870d4c6e6a6d
Normal-9352.png,Normal,Normal/images/Normal-9352.png,Normal/masks/Normal-9352.png,True,G18886,val,f97597ef41a6e5a88a490bc0a959c153,9967560a47386f3a
Normal-9353.png,Normal,Normal/images/Normal-9353.png,Normal/masks/Normal-9353.png,True,G18887,val,66b8f9bf1c31a63b1b948d07f15d2649,997b061e1f0f7898
Normal-9354.png,Normal,Normal/images/Normal-9354.png,Normal/masks/Normal-9354.png,True,G18888,train,af2abc1baa5d6c5bb319f7ca05cac87b,992ec61b444f5bc5
Normal-9355.png,Normal,Normal/images/Normal-9355.png,Normal/masks/Normal-9355.png,True,G18889,train,7ae1c6d058b0070851507036da25fb15,dd2a534b5b4d0b19
Normal-9356.png,Normal,Normal/images/Normal-9356.png,Normal/masks/Normal-9356.png,True,G18890,test,5e1c38283c433d7d2ba9057939e0ebe7,dc2f424f0c6a3de4
Normal-9357.png,Normal,Normal/images/Normal-9357.png,Normal/masks/Normal-9357.png,True,G18891,train,bad5050bbd49a5da6c2d9d32d038d13c,d978071e1a4d3ab9
Normal-9358.png,Normal,Normal/images/Normal-9358.png,Normal/masks/Normal-9358.png,True,G18892,train,8ef316ef877f5eaa4b27cb84a437ecb5,d93b47871e1a4c1d
Normal-9359.png,Normal,Normal/images/Normal-9359.png,Normal/masks/Normal-9359.png,True,G18893,train,f261463899a136933943a10849ca47d7,d42d4acb0f3cfc50
Normal-936.png,Normal,Normal/images/Normal-936.png,Normal/masks/Normal-936.png,True,G18894,train,bb9d7e40c1d8d6c358c31032c749fd55,802ae27a5c59797d
Normal-9360.png,Normal,Normal/images/Normal-9360.png,Normal/masks/Normal-9360.png,True,G18895,val,fe24cb75914865a551ffb41231b2423b,81f9c712855eb86b
Normal-9361.png,Normal,Normal/images/Normal-9361.png,Normal/masks/Normal-9361.png,True,G18896,test,50e3ca2914c6582346d7d10fca00ca48,987dc65a454f1a47
Normal-9362.png,Normal,Normal/images/Normal-9362.png,Normal/masks/Normal-9362.png,True,G18897,test,bf1e6485b52cfc21a3ee95f43cd2b5dc,992b171a591b1b5b
Normal-9363.png,Normal,Normal/images/Normal-9363.png,Normal/masks/Normal-9363.png,True,G18898,train,6973a7129b8825f8cc0d68405355e7c6,d93d46461b194d3d
Normal-9364.png,Normal,Normal/images/Normal-9364.png,Normal/masks/Normal-9364.png,True,G18899,train,fbac0dd51b968253a6e0d26a5406bdd6,dd2d42c25b836f0e
Normal-9365.png,Normal,Normal/images/Normal-9365.png,Normal/masks/Normal-9365.png,True,G18900,train,fdc54cea3c1de6ded3be00df3c7734df,dc2d424b0f4c6e3d
Normal-9366.png,Normal,Normal/images/Normal-9366.png,Normal/masks/Normal-9366.png,True,G18901,train,5936077e422714279591b64f83b5ef03,dd2fc21b464e1acc
Normal-9367.png,Normal,Normal/images/Normal-9367.png,Normal/masks/Normal-9367.png,True,G18902,train,5f4bff1941310b887691fb9146001e1c,d929d5c65e1c4c4b
Normal-9368.png,Normal,Normal/images/Normal-9368.png,Normal/masks/Normal-9368.png,True,G18903,train,fc7075760692f2ca88bbad71f36bbc88,9968061a594f7af9
Normal-9369.png,Normal,Normal/images/Normal-9369.png,Normal/masks/Normal-9369.png,True,G18904,train,c16b4531ff026d69b959918c686de5e7,d92946971bc70ed8
Normal-937.png,Normal,Normal/images/Normal-937.png,Normal/masks/Normal-937.png,True,G18905,train,57bcd7dee76fa811bdecf1a5778b8992,946ea71bcf0f0c4c
Normal-9370.png,Normal,Normal/images/Normal-9370.png,Normal/masks/Normal-9370.png,True,G18906,val,c040634d74fc57f150c230da2b029819,996807161e5f1e5e
Normal-9371.png,Normal,Normal/images/Normal-9371.png,Normal/masks/Normal-9371.png,True,G18907,train,c7447ae22ca9723c8d7141f282a41fab,953d421f470f1e47
Normal-9372.png,Normal,Normal/images/Normal-9372.png,Normal/masks/Normal-9372.png,True,G18908,test,68170c7556e45a765db80340cfece463,887f05125b0d7e5b
Normal-9373.png,Normal,Normal/images/Normal-9373.png,Normal/masks/Normal-9373.png,True,G18909,train,cc9d912c6607d86c867091af19dc2c1d,9928075e5b4c7af8
Normal-9374.png,Normal,Normal/images/Normal-9374.png,Normal/masks/Normal-9374.png,True,G18910,val,3de372121a6ed1c9891b4b85bc39f887,a84dc35b5d4d093d
Normal-9375.png,Normal,Normal/images/Normal-9375.png,Normal/masks/Normal-9375.png,True,G18911,train,e6d93305b84da78e5512838cf9f7e33f,dd28560b5f092abd
Normal-9376.png,Normal,Normal/images/Normal-9376.png,Normal/masks/Normal-9376.png,True,G18912,train,f4cabae4f77903cb0d05072c502108e4,89688c173b5d0f5d
Normal-9377.png,Normal,Normal/images/Normal-9377.png,Normal/masks/Normal-9377.png,True,G18913,train,f89db178b8b13b9c9a05ea67edabbdc9,d92d461a1dec3ba4
Normal-9378.png,Normal,Normal/images/Normal-9378.png,Normal/masks/Normal-9378.png,True,G18914,train,841b8cc61310c169b160bf6ef696d312,942ec30b0aaeaeaf
Normal-9379.png,Normal,Normal/images/Normal-9379.png,Normal/masks/Normal-9379.png,True,G18915,test,dfcbf897321ead746285fd9e93d56a5c,c07f42025d4f1e5f
Normal-938.png,Normal,Normal/images/Normal-938.png,Normal/masks/Normal-938.png,True,G18916,val,c8ce8af14e214801664c555f2e9a80b1,912dc61acd691dcd
Normal-9380.png,Normal,Normal/images/Normal-9380.png,Normal/masks/Normal-9380.png,True,G18917,train,b76cfb5e848209efd4f8f696db3e4a57,997fc61a5949184f
Normal-9381.png,Normal,Normal/images/Normal-9381.png,Normal/masks/Normal-9381.png,True,G18918,test,d37ab9ef668fc1a444315cbf3cee6533,bc4cc30b0d86babb
Normal-9382.png,Normal,Normal/images/Normal-9382.png,Normal/masks/Normal-9382.png,True,G18919,train,d0bb806692e79157725737b6282ca419,996bc61f184c7b38
Normal-9383.png,Normal,Normal/images/Normal-9383.png,Normal/masks/Normal-9383.png,True,G18920,train,f066c117532dc50fabf205cd8a87538f,9169869e3cc67a95
Normal-9384.png,Normal,Normal/images/Normal-9384.png,Normal/masks/Normal-9384.png,True,G18921,train,71965f6b0533c6de5835493cdb3a4082,c139c60e174e7e3a
Normal-9385.png,Normal,Normal/images/Normal-9385.png,Normal/masks/Normal-9385.png,True,G18922,test,ca9bd5afb9d6b9cdb1b05e1a904eddd4,9939c61a59565e39
Normal-9386.png,Normal,Normal/images/Normal-9386.png,Normal/masks/Normal-9386.png,True,G18923,train,1bfc373bf311259518b7e02d9312b376,d43c8b0b464f5c7c
Normal-9387.png,Normal,Normal/images/Normal-9387.png,Normal/masks/Normal-9387.png,True,G18924,train,3f9dd3b28ac8ec65e6c350d69c5fbc0a,883d42535d4b4e7d
Normal-9388.png,Normal,Normal/images/Normal-9388.png,Normal/masks/Normal-9388.png,True,G18925,train,afeef99ee07748be14b50399a0af6f13,917f86161e071e1f
Normal-9389.png,Normal,Normal/images/Normal-9389.png,Normal/masks/Normal-9389.png,True,G18926,train,ce59f41ffc633c03f9eb1b74be0b7ddd,dd2dc2d70bd5021b
Normal-939.png,Normal,Normal/images/Normal-939.png,Normal/masks/Normal-939.png,True,G18927,train,5d4250d3335a8734fe5ac5247c72d6d7,916cc6789d6939c6
Normal-9390.png,Normal,Normal/images/Normal-9390.png,Normal/masks/Normal-9390.png,True,G18928,val,912eff593a0cefd836fb5f7dd1421677,dc0cc34b0e2d6d3d
Normal-9391.png,Normal,Normal/images/Normal-9391.png,Normal/masks/Normal-9391.png,True,G18929,train,5374e435c9f5b49b1b07b18f70526e44,d969061a194d7bb9
Normal-9392.png,Normal,Normal/images/Normal-9392.png,Normal/masks/Normal-9392.png,True,G18930,train,3d0ec53e4e64e92fc1e820858ac11174,9569461e5b4e383b
Normal-9393.png,Normal,Normal/images/Normal-9393.png,Normal/masks/Normal-9393.png,True,G18931,train,5f533e1fbdc0e1c5078e50b6a4ae0717,d96a561a0a1d1f3e
Normal-9394.png,Normal,Normal/images/Normal-9394.png,Normal/masks/Normal-9394.png,True,G18932,train,22e48f7c8a8a257e925aeeb6d09d39c2,897d055e586979b8
Normal-9395.png,Normal,Normal/images/Normal-9395.png,Normal/masks/Normal-9395.png,True,G18933,val,53188b308643858ec1bd0e935b7ecac2,817a865b7d5b1916
Normal-9396.png,Normal,Normal/images/Normal-9396.png,Normal/masks/Normal-9396.png,True,G18934,train,6c8415412ea3a87d2edbfb8c45379969,9929d61fb84ca9b8
Normal-9397.png,Normal,Normal/images/Normal-9397.png,Normal/masks/Normal-9397.png,True,G18935,val,d9d5941c39db2d6d8b5f171860deee12,d03d62434b5bcb4b
Normal-9398.png,Normal,Normal/images/Normal-9398.png,Normal/masks/Normal-9398.png,True,G18936,test,5d4f3e27f7ef53a79c16325700b3c0a8,9969065e5acf7838
Normal-9399.png,Normal,Normal/images/Normal-9399.png,Normal/masks/Normal-9399.png,True,G18937,test,48d4cfed22292220aeaad72899c83da3,d97ad61f540a194b
Normal-94.png,Normal,Normal/images/Normal-94.png,Normal/masks/Normal-94.png,True,G18938,test,723f07827e9ced699039cb4df5988d5d,d03dc71939cc39a6
Normal-940.png,Normal,Normal/images/Normal-940.png,Normal/masks/Normal-940.png,True,G18939,train,09c56a4fea7e54f36328ad293e26a975,912bc21add791dd1
Normal-9400.png,Normal,Normal/images/Normal-9400.png,Normal/masks/Normal-9400.png,True,G18940,train,b912c30172c595dff609a599ff9949f6,d1bdc6bf19843990
Normal-9401.png,Normal,Normal/images/Normal-9401.png,Normal/masks/Normal-9401.png,True,G18941,train,46cd7f2e2ff6c82c0df0570d53012005,91abd6151e0e5d1d
Normal-9402.png,Normal,Normal/images/Normal-9402.png,Normal/masks/Normal-9402.png,True,G18942,train,7706a542d3fbcef009bcb7752bb247a9,d979871618475f19
Normal-9403.png,Normal,Normal/images/Normal-9403.png,Normal/masks/Normal-9403.png,True,G18943,train,ea3cefb3a14a795d74622503937b931a,dc3d424b1dc44f3c
Normal-9404.png,Normal,Normal/images/Normal-9404.png,Normal/masks/Normal-9404.png,True,G18944,train,062650b49d0d5073b78965e260319750,897b870a1a0ddb3b
Normal-9405.png,Normal,Normal/images/Normal-9405.png,Normal/masks/Normal-9405.png,True,G18945,train,a75c90064652b4b62d8ab9a6bd06b9b3,997b861e190f7871
Normal-9406.png,Normal,Normal/images/Normal-9406.png,Normal/masks/Normal-9406.png,True,G18946,train,75a9ebb6b4a3b02545683b51147d95c8,8169d7961a85d67a
Normal-9407.png,Normal,Normal/images/Normal-9407.png,Normal/masks/Normal-9407.png,True,G18947,train,e6c96a7f1f0e1d9e6aaedcc080149a62,dd29419716970f1e
Normal-9408.png,Normal,Normal/images/Normal-9408.png,Normal/masks/Normal-9408.png,True,G18948,val,4f544be753a382b6538888a7f300c635,dd2c061f1e4c7b58
Normal-9409.png,Normal,Normal/images/Normal-9409.png,Normal/masks/Normal-9409.png,True,G18949,train,e3a8a4a5cc2c0ae705c24e0846cf33f3,d83d860b0a4c7f7c
Normal-941.png,Normal,Normal/images/Normal-941.png,Normal/masks/Normal-941.png,True,G18950,train,7e845f0f0a658f7bd7f3f36b03617e9b,907dc65a8b6d38e4
Normal-9410.png,Normal,Normal/images/Normal-9410.png,Normal/masks/Normal-9410.png,True,G18951,train,33631d9cdc529c037dcca6fa22979aa4,8c4dc21f4c1f4f47
Normal-9411.png,Normal,Normal/images/Normal-9411.png,Normal/masks/Normal-9411.png,True,G18952,train,3d1a85b54cf3ec34fee8f51872525d28,9979061a1f4e3ab9
Normal-9412.png,Normal,Normal/images/Normal-9412.png,Normal/masks/Normal-9412.png,True,G18953,train,434f17e4f0079fb481eb40ebca9f38c7,953cc643594e4f1d
Normal-9413.png,Normal,Normal/images/Normal-9413.png,Normal/masks/Normal-9413.png,True,G18954,val,1af9f57ec4581d85077153f78e656fdb,d96d861a954f1947
Normal-9414.png,Normal,Normal/images/Normal-9414.png,Normal/masks/Normal-9414.png,True,G18955,test,c23ce59b4fa988f8f4d370f839488d5a,dd3dc2571bc00f19
Normal-9415.png,Normal,Normal/images/Normal-9415.png,Normal/masks/Normal-9415.png,True,G18956,val,d32cbc443d03e818b3a54f5fc3e8edd7,9969851e1ac75e1e
Normal-9416.png,Normal,Normal/images/Normal-9416.png,Normal/masks/Normal-9416.png,True,G18957,test,c6c616579704de46139e82bdfac75731,c43dc30f1c4b1f4d
Normal-9417.png,Normal,Normal/images/Normal-9417.png,Normal/masks/Normal-9417.png,True,G18958,train,89701e418845cf9b2edb22135021f180,9c2cc71a43531f57
Normal-9418.png,Normal,Normal/images/Normal-9418.png,Normal/masks/Normal-9418.png,True,G18959,train,4657fd85b9643c2fd5e05b3e49bce860,d42f430d2c2a3fb9
Normal-9419.png,Normal,Normal/images/Normal-9419.png,Normal/masks/Normal-9419.png,True,G18960,train,956004b96c76c803e675fe20d41f9a8f,95780f475b594c6c
Normal-942.png,Normal,Normal/images/Normal-942.png,Normal/masks/Normal-942.png,True,G18961,train,b81866f2aee5178faef507b5e5b3168a,c93e4f1e050c3ced
Normal-9420.png,Normal,Normal/images/Normal-9420.png,Normal/masks/Normal-9420.png,True,G18962,train,33707aae31b05f4eb6900f7a54976776,dd6a530f082d2c3f
Normal-9421.png,Normal,Normal/images/Normal-9421.png,Normal/masks/Normal-9421.png,True,G18963,train,881a85391df759366174d3217fbb7e87,dd2a970b5d601ad3
Normal-9422.png,Normal,Normal/images/Normal-9422.png,Normal/masks/Normal-9422.png,True,G18964,val,37e211758b0433bd2eb35351e5465591,d43f530a5e4f180f
Normal-9423.png,Normal,Normal/images/Normal-9423.png,Normal/masks/Normal-9423.png,True,G18965,train,b0f9e8279c942a118b0e41ebe868dde0,c93d861a1d4f5b19
Normal-9424.png,Normal,Normal/images/Normal-9424.png,Normal/masks/Normal-9424.png,True,G18966,train,7eb088538da306532545496c9c37a7c5,8179469e1e8f5b98
Normal-9425.png,Normal,Normal/images/Normal-9425.png,Normal/masks/Normal-9425.png,True,G18967,train,5d293bdb6e7654b2c63ddf7c19d8feb5,d87fc51e16451e0d
Normal-9426.png,Normal,Normal/images/Normal-9426.png,Normal/masks/Normal-9426.png,True,G18968,train,27d76952bbc6afd2a615e796a96c43e1,991d071a495b1ecf
Normal-9427.png,Normal,Normal/images/Normal-9427.png,Normal/masks/Normal-9427.png,True,G18969,test,b33266827137260c6085f616b30d160b,c07e060c5dedeec4
Normal-9428.png,Normal,Normal/images/Normal-9428.png,Normal/masks/Normal-9428.png,True,G18970,val,3cfe1ff9532c3ce93566081de875c369,d82dc21f914f1f91
Normal-9429.png,Normal,Normal/images/Normal-9429.png,Normal/masks/Normal-9429.png,True,G18971,test,380dcf18751252d0374a51204d81fee3,d87dc41f954a1f42
Normal-943.png,Normal,Normal/images/Normal-943.png,Normal/masks/Normal-943.png,True,G18972,train,1f573c8c9101fd7be1bda0e4abbe3083,907ec73e8d1d388c
Normal-9430.png,Normal,Normal/images/Normal-9430.png,Normal/masks/Normal-9430.png,True,G18973,train,fcd6fa012ab0e0394de6708a778d0846,dd2cd20b0e5f0a3d
Normal-9431.png,Normal,Normal/images/Normal-9431.png,Normal/masks/Normal-9431.png,True,G18974,test,e28875b6e1184f7a5322821d19e05e12,d83905961d687fb8
Normal-9432.png,Normal,Normal/images/Normal-9432.png,Normal/masks/Normal-9432.png,True,G18975,train,36908cd472add23b43b453d21a1de754,897c87121f4f5e58
Normal-9433.png,Normal,Normal/images/Normal-9433.png,Normal/masks/Normal-9433.png,True,G18976,test,75b7ec8211ba134ee259370ed933f261,a93986061b195f5f
Normal-9434.png,Normal,Normal/images/Normal-9434.png,Normal/masks/Normal-9434.png,True,G18977,train,830e896369a476c7c720b6ebe8d03f1d,dc7dc34b4a4b0d0d
Normal-9435.png,Normal,Normal/images/Normal-9435.png,Normal/masks/Normal-9435.png,True,G18978,train,17d8b4828b9a18dd30b622dc945f41fc,997b871a1a073a79
Normal-9436.png,Normal,Normal/images/Normal-9436.png,Normal/masks/Normal-9436.png,True,G18979,train,159d614cf0c82e8fb418d5826cdcac95,987f8517165e1f18
Normal-9437.png,Normal,Normal/images/Normal-9437.png,Normal/masks/Normal-9437.png,True,G18980,val,7c2316b682b70b2f97fff2287bdc559a,d13956464f1a1e4f
Normal-9438.png,Normal,Normal/images/Normal-9438.png,Normal/masks/Normal-9438.png,True,G18981,train,cfe3869f55aa854c7ba7ec06b008d0cc,99fbd618565b184a
Normal-9439.png,Normal,Normal/images/Normal-9439.png,Normal/masks/Normal-9439.png,True,G18982,train,76dfa373862c5d5e441a4ffd12eb84d2,dd2cc21b0f0f1d4d
Normal-944.png,Normal,Normal/images/Normal-944.png,Normal/masks/Normal-944.png,True,G18983,train,de069cbf882ef2649d99d7d5aa4cedb0,812bc25ad8597dd9
Normal-9440.png,Normal,Normal/images/Normal-9440.png,Normal/masks/Normal-9440.png,True,G18984,train,a239e8ba459ae4d0414fb8b127d05be5,cd3c420b1c4d3fbc
Normal-9441.png,Normal,Normal/images/Normal-9441.png,Normal/masks/Normal-9441.png,True,G18985,train,33ea33bc5bbc9a86760dfc8602d21b1a,dcaec34d574d1e40
Normal-9442.png,Normal,Normal/images/Normal-9442.png,Normal/masks/Normal-9442.png,True,G18986,val,ee00910fdae48957369e8c086d22ff7f,992816595a6d79e9
Normal-9443.png,Normal,Normal/images/Normal-9443.png,Normal/masks/Normal-9443.png,True,G18987,train,a2741ac0db8d25c57e759dea17d3dfd7,d02bd65549495d5d
Normal-9444.png,Normal,Normal/images/Normal-9444.png,Normal/masks/Normal-9444.png,True,G18988,train,4a5d7d2a03dcb8bfb71ccfa236afc095,817b07869e1e1c5f
Normal-9445.png,Normal,Normal/images/Normal-9445.png,Normal/masks/Normal-9445.png,True,G18989,train,a80fadc465c2c649dd7e71edaeac4f43,8979061e571e195f
Normal-9446.png,Normal,Normal/images/Normal-9446.png,Normal/masks/Normal-9446.png,True,G18990,train,e5b266d0d18c4d06192bbc3ff3a1f6d4,8178174a1b3d6d3b
Normal-9447.png,Normal,Normal/images/Normal-9447.png,Normal/masks/Normal-9447.png,True,G18991,train,ec5bde3582b2dffccecd0426b8aeaa5a,d96b861e445f184f
Normal-9448.png,Normal,Normal/images/Normal-9448.png,Normal/masks/Normal-9448.png,True,G18992,val,18b80265d50c58d5c0ba79210d7334aa,8979071e18595f79
Normal-9449.png,Normal,Normal/images/Normal-9449.png,Normal/masks/Normal-9449.png,True,G18993,val,de43501989aae1301fc5212ead9457af,982957821f5d4a5f
Normal-945.png,Normal,Normal/images/Normal-945.png,Normal/masks/Normal-945.png,True,G18994,train,3fe5038406b0fea0ca5a6fed6cf3ce14,982dc71ac76a3c65
Normal-9450.png,Normal,Normal/images/Normal-9450.png,Normal/masks/Normal-9450.png,True,G18995,train,06491008e10327165bc66c904209d09a,997f061a18ed39b8
Normal-9451.png,Normal,Normal/images/Normal-9451.png,Normal/masks/Normal-9451.png,True,G18996,test,8203a6c67aa32fceaa40a3ec8f0db6b0,dd2e42c31b00adbf
Normal-9452.png,Normal,Normal/images/Normal-9452.png,Normal/masks/Normal-9452.png,True,G18997,test,d841d472b9316430d1ca65b762db80a7,c838db5d984eb83c
Normal-9453.png,Normal,Normal/images/Normal-9453.png,Normal/masks/Normal-9453.png,True,G18998,test,117a97c1eb739ffbc13926990d0fa817,dc7dc30b4c0f0e2d
Normal-9454.png,Normal,Normal/images/Normal-9454.png,Normal/masks/Normal-9454.png,True,G18999,train,89b5ee1a28789a45958e4ef71ca44952,c13dc2471d92cb2f
Normal-9455.png,Normal,Normal/images/Normal-9455.png,Normal/masks/Normal-9455.png,True,G19000,train,d2deac74479a2dcacf5492a8fdff0358,887dc65b0d4e5947
Normal-9456.png,Normal,Normal/images/Normal-9456.png,Normal/masks/Normal-9456.png,True,G19001,train,0531bc19cef9c3d4c7c3611b30cf1b2c,993bc71e184d1e78
Normal-9457.png,Normal,Normal/images/Normal-9457.png,Normal/masks/Normal-9457.png,True,G19002,train,dd23fc238ff998ad51e57752d1ce7cb2,886c46d37f842bb5
Normal-9458.png,Normal,Normal/images/Normal-9458.png,Normal/masks/Normal-9458.png,True,G19003,train,ae9f17d7045b36a4e7b0ac7401c48664,dc2d430f0e2c2d3f
Normal-9459.png,Normal,Normal/images/Normal-9459.png,Normal/masks/Normal-9459.png,True,G19004,train,21e77948b26f8fd60fb0334212e9a4b0,99681e7a78e5e185
Normal-946.png,Normal,Normal/images/Normal-946.png,Normal/masks/Normal-946.png,True,G19005,train,9b7778ba334372ee6399340049440bd9,9168ce7a8979194f
Normal-9460.png,Normal,Normal/images/Normal-9460.png,Normal/masks/Normal-9460.png,True,G19006,train,e538456e43545f867420cc4bf1ec0c96,d969871e1a4f1a4d
Normal-9461.png,Normal,Normal/images/Normal-9461.png,Normal/masks/Normal-9461.png,True,G19007,train,f2137da7094fea975e0cd821c71ed420,9b698e16120e5f3b
Normal-9462.png,Normal,Normal/images/Normal-9462.png,Normal/masks/Normal-9462.png,True,G19008,train,39796be3a3e91bb4a80fa9ef88ad90ff,d92907561a5c7b5a
Normal-9463.png,Normal,Normal/images/Normal-9463.png,Normal/masks/Normal-9463.png,True,G19009,train,6927caad2ffdaceaf904d851a1a73a74,997b851e194d1b39
Normal-9464.png,Normal,Normal/images/Normal-9464.png,Normal/masks/Normal-9464.png,True,G19010,train,9ec55b63fac620d05d6d659e5c27b33a,d07fd65258597138
Normal-9465.png,Normal,Normal/images/Normal-9465.png,Normal/masks/Normal-9465.png,True,G19011,train,c652f1370eadc5329ad07bf09551fb0e,d939061e556e3ac9
Normal-9466.png,Normal,Normal/images/Normal-9466.png,Normal/masks/Normal-9466.png,True,G19012,train,5f48e1d56e79afa15bde77bc56f8ac9b,997e821b0a873b79
Normal-9467.png,Normal,Normal/images/Normal-9467.png,Normal/masks/Normal-9467.png,True,G19013,train,9bb599682829a9a931fc08e145df55fb,d13b46195f2d38a5
Normal-9468.png,Normal,Normal/images/Normal-9468.png,Normal/masks/Normal-9468.png,True,G19014,test,86e86ab7c9828bfcd4ca6414e85c6a93,d42e43431f4e6e6c
Normal-9469.png,Normal,Normal/images/Normal-9469.png,Normal/masks/Normal-9469.png,True,G19015,train,1e4092115d3d88b97484fcfda6ef4fd0,984f821b4d3869bf
Normal-947.png,Normal,Normal/images/Normal-947.png,Normal/masks/Normal-947.png,True,G19016,val,d0ce181b5f87bda63103181d0c653e59,816f8b1b415370f7
Normal-9470.png,Normal,Normal/images/Normal-9470.png,Normal/masks/Normal-9470.png,True,G19017,val,99effa61daf40a1ff6a9b566c1d46e10,d42d037d2867b3d2
Normal-9471.png,Normal,Normal/images/Normal-9471.png,Normal/masks/Normal-9471.png,True,G19018,val,bab557e68f33de78fd9ea365d1c488fd,d939d65e1909591d
Normal-9472.png,Normal,Normal/images/Normal-9472.png,Normal/masks/Normal-9472.png,True,G19019,test,5bcdbbdd369304370806beca6e0a8c80,91795e141f7a3949
Normal-9473.png,Normal,Normal/images/Normal-9473.png,Normal/masks/Normal-9473.png,True,G19020,test,c26f37c8ce46c7e4c269c057728645b7,81615e5e7a786969
Normal-9474.png,Normal,Normal/images/Normal-9474.png,Normal/masks/Normal-9474.png,True,G19021,train,a9f964f477c5cb33fb6f795185b9d7ef,dc39db4b8ba8a8a4
Normal-9475.png,Normal,Normal/images/Normal-9475.png,Normal/masks/Normal-9475.png,True,G19022,train,e2ea5dffe3f6ce07bc418ca87b4dd6f2,d96dc65a594d5819
Normal-9476.png,Normal,Normal/images/Normal-9476.png,Normal/masks/Normal-9476.png,True,G19023,train,cf726902eb8ac2e35714789a3237a0e7,81780716985bdd7b
Normal-9477.png,Normal,Normal/images/Normal-9477.png,Normal/masks/Normal-9477.png,True,G19024,train,adafd06a856dfdf58ebf9bf76bad137e,d83fc319574d184d
Normal-9478.png,Normal,Normal/images/Normal-9478.png,Normal/masks/Normal-9478.png,True,G19025,train,f775de95a5f81061fc206d8c9e5cf810,d429c30d1c2d3e5f
Normal-9479.png,Normal,Normal/images/Normal-9479.png,Normal/masks/Normal-9479.png,True,G19026,train,dd92889e13bacac504505d8ee269cee6,816f871a055f1d4f
Normal-948.png,Normal,Normal/images/Normal-948.png,Normal/masks/Normal-948.png,True,G19027,train,93b045b9dd850436f60547f2040d2caa,c12dc74a9cc9199f
Normal-9480.png,Normal,Normal/images/Normal-9480.png,Normal/masks/Normal-9480.png,True,G19028,train,1d923e5d3555226cbd48f8e5d94f6611,897b07061e171f1f
Normal-9481.png,Normal,Normal/images/Normal-9481.png,Normal/masks/Normal-9481.png,True,G19029,train,614036e43a4627ddd6474ca62772574b,dd3dc21f544e1d88
Normal-9482.png,Normal,Normal/images/Normal-9482.png,Normal/masks/Normal-9482.png,True,G19030,train,c2bd622ba5c7633474e731766c1880b7,91795a1e191bb939
Normal-9483.png,Normal,Normal/images/Normal-9483.png,Normal/masks/Normal-9483.png,True,G19031,train,b8f30fad88e567121d30f547da9d3b3a,cd3d42059f4a2b3d
Normal-9484.png,Normal,Normal/images/Normal-9484.png,Normal/masks/Normal-9484.png,True,G19032,train,9b5eec0ad7b5ace57b11b94a48044d65,d93dc21b546d39b0
Normal-9485.png,Normal,Normal/images/Normal-9485.png,Normal/masks/Normal-9485.png,True,G19033,test,6b6940fa0d4a623aaa18594ff531b38f,817c065a795f78e8
Normal-9486.png,Normal,Normal/images/Normal-9486.png,Normal/masks/Normal-9486.png,True,G19034,val,563820543ef67caab22f94eb20129b29,9d2a831f4d4c5c6d
Normal-9487.png,Normal,Normal/images/Normal-9487.png,Normal/masks/Normal-9487.png,True,G19035,val,534d8df243cc608f72db63b8990d503d,9169161e1b5e7939
Normal-9488.png,Normal,Normal/images/Normal-9488.png,Normal/masks/Normal-9488.png,True,G19036,val,1dbdffde7b6b117a17b475f377a5d71a,cc2dd3030f535e1e
Normal-9489.png,Normal,Normal/images/Normal-9489.png,Normal/masks/Normal-9489.png,True,G19037,test,03c4f3d4a31a8a0b2f5543f3a2f78263,8c3e424b1d4f0f1f
Normal-949.png,Normal,Normal/images/Normal-949.png,Normal/masks/Normal-949.png,True,G19038,train,d8c90419917da7f4bf841b6faad882ed,812fc65af84b29cd
Normal-9490.png,Normal,Normal/images/Normal-9490.png,Normal/masks/Normal-9490.png,True,G19039,train,98f8fcef02255501a42262bbe14e75ce,913b161a5be9b9b0
Normal-9491.png,Normal,Normal/images/Normal-9491.png,Normal/masks/Normal-9491.png,True,G19040,train,1b28d2ebbd772aae09ce5c209cfefd88,d83b85071707571f
Normal-9492.png,Normal,Normal/images/Normal-9492.png,Normal/masks/Normal-9492.png,True,G19041,train,32fa9191f1e15435f7839fbf6a145305,dd3d86021b465f39
Normal-9493.png,Normal,Normal/images/Normal-9493.png,Normal/masks/Normal-9493.png,True,G19042,train,10e981671990f741406cf388a1756d14,d52ec61b0bcf0a3a
Normal-9494.png,Normal,Normal/images/Normal-9494.png,Normal/masks/Normal-9494.png,True,G19043,train,ed6ec4caa6178ef2824c9b267305e953,9979c61e551b194d
Normal-9495.png,Normal,Normal/images/Normal-9495.png,Normal/masks/Normal-9495.png,True,G19044,train,7dc187003f2f478f2f4466565ef52247,817b041e1f595d5d
Normal-9496.png,Normal,Normal/images/Normal-9496.png,Normal/masks/Normal-9496.png,True,G19045,train,c7c27f467e7197352987f40f703e3a11,817b871617971e17
Normal-9497.png,Normal,Normal/images/Normal-9497.png,Normal/masks/Normal-9497.png,True,G19046,train,1fb1094579d90b06979c7dd62cca8ae8,9570b4d6933a0d5b
Normal-9498.png,Normal,Normal/images/Normal-9498.png,Normal/masks/Normal-9498.png,True,G19047,test,c546bdc5d32991ad9ad3ee1b4925fe66,996f46863bc17ec0
Normal-9499.png,Normal,Normal/images/Normal-9499.png,Normal/masks/Normal-9499.png,True,G19048,train,d3c9ddd2bcb23ef2aa28f373c923903d,d538d34b0e2c6d3c
Normal-95.png,Normal,Normal/images/Normal-95.png,Normal/masks/Normal-95.png,True,G19049,train,2b824dbd1798cbbd8c612462f324d0d4,906fe63acf382c2c
Normal-950.png,Normal,Normal/images/Normal-950.png,Normal/masks/Normal-950.png,True,G19050,train,2afac4f2c080a5114cd5f292d07d50bf,986da71e8b1f0a0f
Normal-9500.png,Normal,Normal/images/Normal-9500.png,Normal/masks/Normal-9500.png,True,G19051,train,d3009e43f100eb62624ea496632df41e,c83dc34b5e4b0d1e
Normal-9501.png,Normal,Normal/images/Normal-9501.png,Normal/masks/Normal-9501.png,True,G19052,train,3ac6d3fbee504538b5feaed941ffe487,9969861a195f5a79
Normal-9502.png,Normal,Normal/images/Normal-9502.png,Normal/masks/Normal-9502.png,True,G19053,train,99a64a9ad71086411589af75321e02f7,997dc5561e491b1c
Normal-9503.png,Normal,Normal/images/Normal-9503.png,Normal/masks/Normal-9503.png,True,G19054,train,c307c3545140295f53c7f2edd8c7797b,cc3dd3530d8a4b0f
Normal-9504.png,Normal,Normal/images/Normal-9504.png,Normal/masks/Normal-9504.png,True,G19055,train,958569f6188b5d9953df5dde477283b7,d97b461a1d053b55
Normal-9505.png,Normal,Normal/images/Normal-9505.png,Normal/masks/Normal-9505.png,True,G19056,train,ddef46d91e192b77d5113d19c56a2121,997984161a5d5f78
Normal-9506.png,Normal,Normal/images/Normal-9506.png,Normal/masks/Normal-9506.png,True,G19057,val,40181736ada3dfe115957c7e79959228,8939c61e194d6b3d
Normal-9507.png,Normal,Normal/images/Normal-9507.png,Normal/masks/Normal-9507.png,True,G19058,val,6d69e30d6a099c454a29cb666d322d14,d939061d1e4e157b
Normal-9508.png,Normal,Normal/images/Normal-9508.png,Normal/masks/Normal-9508.png,True,G19059,val,ce0b071b307d5a323eff09ec8fe6e9b0,89ff8612875b015f
Normal-9509.png,Normal,Normal/images/Normal-9509.png,Normal/masks/Normal-9509.png,True,G19060,test,a4f1a11b46dcde454ae6b0e95411e932,dc3dd2531b820f1e
Normal-951.png,Normal,Normal/images/Normal-951.png,Normal/masks/Normal-951.png,True,G19061,test,be09901e6a1822787f75444f440e7e15,917847104e79797d
Normal-9510.png,Normal,Normal/images/Normal-9510.png,Normal/masks/Normal-9510.png,True,G19062,train,7d9493c075257cf119942a44e8324453,887e070a5b4d7a3d
Normal-9511.png,Normal,Normal/images/Normal-9511.png,Normal/masks/Normal-9511.png,True,G19063,train,5eb14dd06652a5f3e5228fd2eb1f7360,887bd61a1903bf96
Normal-9512.png,Normal,Normal/images/Normal-9512.png,Normal/masks/Normal-9512.png,True,G19064,train,c71a9a436ef9e35e5bce3d3a93d57470,dc2e065b1a5c6c7c
Normal-9513.png,Normal,Normal/images/Normal-9513.png,Normal/masks/Normal-9513.png,True,G19065,train,35bad76af5f94199197ede9ee9a4e595,d42e071d6928fbc3
Normal-9514.png,Normal,Normal/images/Normal-9514.png,Normal/masks/Normal-9514.png,True,G19066,train,3cdfa51880219cfa5cdf7d6804257d54,d97944561f174a1b
Normal-9515.png,Normal,Normal/images/Normal-9515.png,Normal/masks/Normal-9515.png,True,G19067,val,6d1f350f5be721b5912a0117209eef32,cc2e524b0f646b3d
Normal-9516.png,Normal,Normal/images/Normal-9516.png,Normal/masks/Normal-9516.png,True,G19068,train,4465992b4502e632883fd4ace0c56ec8,b97d861a194d394d
Normal-9517.png,Normal,Normal/images/Normal-9517.png,Normal/masks/Normal-9517.png,True,G19069,train,f62a92c72c53d38c6c764bf0d001032a,887fd61ac41a5d5a
Normal-9518.png,Normal,Normal/images/Normal-9518.png,Normal/masks/Normal-9518.png,True,G19070,val,bb64edb2304d1cfa3d8a6510e9dda7c0,887b561e442f392f
Normal-9519.png,Normal,Normal/images/Normal-9519.png,Normal/masks/Normal-9519.png,True,G19071,train,2eda42ee28eb61985888e725e81a0a22,cc3d430ace4f3ce4
Normal-952.png,Normal,Normal/images/Normal-952.png,Normal/masks/Normal-952.png,True,G19072,train,b358d857be4ce2d1c1cea6ac311e454d,857f4f5b584518c6
Normal-9520.png,Normal,Normal/images/Normal-9520.png,Normal/masks/Normal-9520.png,True,G19073,train,3caceea77dbad427f5ae4d3d0e7d5318,d979045e184d5f1d
Normal-9521.png,Normal,Normal/images/Normal-9521.png,Normal/masks/Normal-9521.png,True,G19074,val,adc901e3e623f351781177f3702e6b1e,dd3d06191c4e7978
Normal-9522.png,Normal,Normal/images/Normal-9522.png,Normal/masks/Normal-9522.png,True,G19075,test,dbcd7e9b2317257e8c0ddf55bc97f3e7,d635d14957d10d87
Normal-9523.png,Normal,Normal/images/Normal-9523.png,Normal/masks/Normal-9523.png,True,G19076,test,7597301e80217db4eec9e72259f0d325,89790786961aeb1f
Normal-9524.png,Normal,Normal/images/Normal-9524.png,Normal/masks/Normal-9524.png,True,G19077,train,90d930bd162e2289a4b1d98285cdf92c,897d061a595f595a
Normal-9525.png,Normal,Normal/images/Normal-9525.png,Normal/masks/Normal-9525.png,True,G19078,test,1ee57e532c3165610b584e3cbbfe1348,8979d61e191e1f38
Normal-9526.png,Normal,Normal/images/Normal-9526.png,Normal/masks/Normal-9526.png,True,G19079,train,93732ecc52ae7b84f5fc5777959921f2,dd6e475d044c4c4f
Normal-9527.png,Normal,Normal/images/Normal-9527.png,Normal/masks/Normal-9527.png,True,G19080,train,7d90c73ce26b8f2d3faa41e346cce259,c179101e595b7b39
Normal-9528.png,Normal,Normal/images/Normal-9528.png,Normal/masks/Normal-9528.png,True,G19081,train,10674407bdf51542a8c41d0e27b5f999,9969861a5d5f18c7
Normal-9529.png,Normal,Normal/images/Normal-9529.png,Normal/masks/Normal-9529.png,True,G19082,train,7f41a3212a522ad1325700faa6ee98d1,9c6cc6571b560f58
Normal-953.png,Normal,Normal/images/Normal-953.png,Normal/masks/Normal-953.png,True,G19083,train,b49ce2b40aeaff11763488ba6c506729,912f8f1a19455cd7
Normal-9530.png,Normal,Normal/images/Normal-9530.png,Normal/masks/Normal-9530.png,True,G19084,train,b245479ff5edf3950414adf7f8a1553b,8978165b595a69e5
Normal-9531.png,Normal,Normal/images/Normal-9531.png,Normal/masks/Normal-9531.png,True,G19085,train,96d7152a523e329dfba2ce41b4cc448e,9128071e5b7e7a78
Normal-9532.png,Normal,Normal/images/Normal-9532.png,Normal/masks/Normal-9532.png,True,G19086,train,7cd2a30277ed3c58987c5d17fe7a5e26,d42e534b4a6f0a3d
Normal-9533.png,Normal,Normal/images/Normal-9533.png,Normal/masks/Normal-9533.png,True,G19087,train,e5ecb2560221148dd87f2ef9f4084c25,987b571a18097b3d
Normal-9534.png,Normal,Normal/images/Normal-9534.png,Normal/masks/Normal-9534.png,True,G19088,train,7183bc3949eeda8e90b6419aeba30eee,8968161d7f7879e0
Normal-9535.png,Normal,Normal/images/Normal-9535.png,Normal/masks/Normal-9535.png,True,G19089,val,907c3a8ff7e04a40aec5a1605e3de392,9179d65a194d591b
Normal-9536.png,Normal,Normal/images/Normal-9536.png,Normal/masks/Normal-9536.png,True,G19090,train,293749e85c9a04e896a3dd3e1a3b06f4,917cc6591f4e3919
Normal-9537.png,Normal,Normal/images/Normal-9537.png,Normal/masks/Normal-9537.png,True,G19091,train,cd445fde1ad563163200ca8c00b0283a,8a3de606bc9aa5e1
Normal-9538.png,Normal,Normal/images/Normal-9538.png,Normal/masks/Normal-9538.png,True,G19092,train,fc18bd1d69481ee759a15bbc19a5f4c1,dd2ad71a5d441b19
Normal-9539.png,Normal,Normal/images/Normal-9539.png,Normal/masks/Normal-9539.png,True,G19093,train,41aaf168c9f5ac0314fff84ad6a99e0f,8168161f1b5f6979
Normal-954.png,Normal,Normal/images/Normal-954.png,Normal/masks/Normal-954.png,True,G19094,val,f93a01d2fd8931f355399087c46a5d77,807fce7a5949588d
Normal-9540.png,Normal,Normal/images/Normal-9540.png,Normal/masks/Normal-9540.png,True,G19095,val,be57e7ca23672ef5b5a55396377b989e,8168848632d75f7f
Normal-9541.png,Normal,Normal/images/Normal-9541.png,Normal/masks/Normal-9541.png,True,G19096,test,ef92ff9fef143036af11a3568c7b4620,997d871a1c4d5ae8
Normal-9542.png,Normal,Normal/images/Normal-9542.png,Normal/masks/Normal-9542.png,True,G19097,test,0036f50ccfaf7e07031407153edaee12,d13f46461b4a5f38
Normal-9543.png,Normal,Normal/images/Normal-9543.png,Normal/masks/Normal-9543.png,True,G19098,train,ddaafc184c3d585d4b64dd8fa559f985,c83d471b38ed38f0
Normal-9544.png,Normal,Normal/images/Normal-9544.png,Normal/masks/Normal-9544.png,True,G19099,test,487e2ffedb3dca707f3df201c2a6ebff,dd2fc21f0be11a0b
Normal-9545.png,Normal,Normal/images/Normal-9545.png,Normal/masks/Normal-9545.png,True,G19100,train,a99e7200a36a18228cd3ae0fd66e1795,857c0f061d5d6d0f
Normal-9546.png,Normal,Normal/images/Normal-9546.png,Normal/masks/Normal-9546.png,True,G19101,train,085b99ed832182393983b4c045c7fd80,993f461b4b0b196d
Normal-9547.png,Normal,Normal/images/Normal-9547.png,Normal/masks/Normal-9547.png,True,G19102,train,37015f72527a2915f71c0cdde28b8a61,d939c3135d442b3b
Normal-9548.png,Normal,Normal/images/Normal-9548.png,Normal/masks/Normal-9548.png,True,G19103,train,c32da4e92eada4c50e9bbda29604d064,d42f42570b4a6f6c
Normal-9549.png,Normal,Normal/images/Normal-9549.png,Normal/masks/Normal-9549.png,True,G19104,train,b4fdec8cea5ca36d7a32da53cc909a49,dd39c2439d422d79
Normal-955.png,Normal,Normal/images/Normal-955.png,Normal/masks/Normal-955.png,True,G19105,train,905cf269a5030e699590ec33b2a4b210,9069c61ed8599ccf
Normal-9550.png,Normal,Normal/images/Normal-9550.png,Normal/masks/Normal-9550.png,True,G19106,train,6919223117270239143bc39c05ff68e9,887f871e1a695d38
Normal-9551.png,Normal,Normal/images/Normal-9551.png,Normal/masks/Normal-9551.png,True,G19107,train,d334e81164034aecd2424b43ab378bda,c83d425f1d4a0f3d
Normal-9552.png,Normal,Normal/images/Normal-9552.png,Normal/masks/Normal-9552.png,True,G19108,train,2b168b731dc053107bec8318372b2889,d97d061a464e3d4e
Normal-9553.png,Normal,Normal/images/Normal-9553.png,Normal/masks/Normal-9553.png,True,G19109,train,f660accff4a22e6222e2697309d6e884,d42dc34f0e0bbd1c
Normal-9554.png,Normal,Normal/images/Normal-9554.png,Normal/masks/Normal-9554.png,True,G19110,train,af26d5d7b196e70d7b95d90e70bb5623,897b851e58417f59
Normal-9555.png,Normal,Normal/images/Normal-9555.png,Normal/masks/Normal-9555.png,True,G19111,val,d9fcce4af466e9c083a7f367ea7cd55d,d90a86181d5f5e5e
Normal-9556.png,Normal,Normal/images/Normal-9556.png,Normal/masks/Normal-9556.png,True,G19112,val,c5477d4426c607f4d8297465bf54be6e,d42f56171e045f1b
Normal-9557.png,Normal,Normal/images/Normal-9557.png,Normal/masks/Normal-9557.png,True,G19113,train,2122f1783d04745a57696f0835383d55,dc3dc34b54c20b9e
Normal-9558.png,Normal,Normal/images/Normal-9558.png,Normal/masks/Normal-9558.png,True,G19114,val,1d99f93a789d3431cb7c39cd8af4a4e6,dc2e821f4c5a786d
Normal-9559.png,Normal,Normal/images/Normal-9559.png,Normal/masks/Normal-9559.png,True,G19115,test,5b87f72b332353a6253957db01b117dd,dd2c521b470f3d26
Normal-956.png,Normal,Normal/images/Normal-956.png,Normal/masks/Normal-956.png,True,G19116,train,22796a87818b63d3e7415eed3a4747c9,d02fce1a5c4d1dd4
Normal-9560.png,Normal,Normal/images/Normal-9560.png,Normal/masks/Normal-9560.png,True,G19117,test,6320b54939d7e4a5d01f9eb7c9695bcd,83719d35335477c1
Normal-9561.png,Normal,Normal/images/Normal-9561.png,Normal/masks/Normal-9561.png,True,G19118,train,0d991259e2d44aa61e8fa089d55d4d77,d42e53534b4a0f4f
Normal-9562.png,Normal,Normal/images/Normal-9562.png,Normal/masks/Normal-9562.png,True,G19119,train,61dff20e9affd61d1f8c45ebbe2abcd0,dd3cd2030f860f9d
Normal-9563.png,Normal,Normal/images/Normal-9563.png,Normal/masks/Normal-9563.png,True,G19120,train,d6389e820acca72fd1e2ca8ad148dbcc,807fc7570a5f005f
Normal-9564.png,Normal,Normal/images/Normal-9564.png,Normal/masks/Normal-9564.png,True,G19121,train,7b704925063a738e44b84d5b15634dbb,9c2e03194f1f0f4f
Normal-9565.png,Normal,Normal/images/Normal-9565.png,Normal/masks/Normal-9565.png,True,G19122,val,4d2e6e16958fbd95a2a80fe42a247cd0,8178869e1f5f1a59
Normal-9566.png,Normal,Normal/images/Normal-9566.png,Normal/masks/Normal-9566.png,True,G19123,test,0a11c2a526ab814739aa314588482386,897d86de945e1a15
Normal-9567.png,Normal,Normal/images/Normal-9567.png,Normal/masks/Normal-9567.png,True,G19124,train,e8d518814d7ab58a84b996ff5811fa5b,dc2ac34f0d0e2e7c
Normal-9568.png,Normal,Normal/images/Normal-9568.png,Normal/masks/Normal-9568.png,True,G19125,train,3f85f9a0e31413fc80b366c0352a68a0,dd2ec2470f4c6f18
Normal-9569.png,Normal,Normal/images/Normal-9569.png,Normal/masks/Normal-9569.png,True,G19126,train,348411d125a1441ef1855702753152dc,cc3dc24b1f4a4d3c
Normal-957.png,Normal,Normal/images/Normal-957.png,Normal/masks/Normal-957.png,True,G19127,train,8a2187d023f117f34205a64921dd3b24,902fce1ada491dd6
Normal-9570.png,Normal,Normal/images/Normal-9570.png,Normal/masks/Normal-9570.png,True,G19128,val,442f18291ddbbf65ffb4e2c32447e1f1,987dc1961acb1e5a
Normal-9571.png,Normal,Normal/images/Normal-9571.png,Normal/masks/Normal-9571.png,True,G19129,train,d0970cc805ab6e3b758809d05d44d18d,8979c6d3184f5b1a
Normal-9572.png,Normal,Normal/images/Normal-9572.png,Normal/masks/Normal-9572.png,True,G19130,train,b1616bae52503ff20be7bbb8d91eb88d,d93ed05b1e461b1c
Normal-9573.png,Normal,Normal/images/Normal-9573.png,Normal/masks/Normal-9573.png,True,G19131,train,c8334c5da08d17fc8f88568be094e10b,c46f434e0ee62de4
Normal-9574.png,Normal,Normal/images/Normal-9574.png,Normal/masks/Normal-9574.png,True,G19132,train,2474ddbbd71d79c1c1ec8b85a23de442,996f06386927e4e6
Normal-9575.png,Normal,Normal/images/Normal-9575.png,Normal/masks/Normal-9575.png,True,G19133,test,e73f48240afa6b2c91e6c20de480d145,dd3dc25b1d404b39
Normal-9576.png,Normal,Normal/images/Normal-9576.png,Normal/masks/Normal-9576.png,True,G19134,train,860242beb5742a83181d5b53d30a5507,c83de303198bcb2f
Normal-9577.png,Normal,Normal/images/Normal-9577.png,Normal/masks/Normal-9577.png,True,G19135,val,a98edd07c2e004454b27dbe3cbe4d751,d42f525f0ec65b0c
Normal-9578.png,Normal,Normal/images/Normal-9578.png,Normal/masks/Normal-9578.png,True,G19136,test,cb34533043216c0c504fde2c71ec1104,997f87121e453a1e
Normal-9579.png,Normal,Normal/images/Normal-9579.png,Normal/masks/Normal-9579.png,True,G19137,test,fa43d53d04b8edf1e2076e60730597da,897945861a598f5f
Normal-958.png,Normal,Normal/images/Normal-958.png,Normal/masks/Normal-958.png,True,G19138,train,b9ebad3aec29cb1894d7d50ed5394f56,917c863e8d2d2cce
Normal-9580.png,Normal,Normal/images/Normal-9580.png,Normal/masks/Normal-9580.png,True,G19139,test,b4ab93ca62265c6213666e6d657d5ece,9979c71685c63aa5
Normal-9581.png,Normal,Normal/images/Normal-9581.png,Normal/masks/Normal-9581.png,True,G19140,train,b0a3db9be2d885b9dddb6cd0e5915e0e,d43d43c30c1cedbc
Normal-9582.png,Normal,Normal/images/Normal-9582.png,Normal/masks/Normal-9582.png,True,G19141,train,ab000af2d65c2603f49a31f651df6998,dc2c424b4f4e2d1f
Normal-9583.png,Normal,Normal/images/Normal-9583.png,Normal/masks/Normal-9583.png,True,G19142,train,e3e46c53de74d029847d2f06c3f83fe6,897c061a1d5a5f5e
Normal-9584.png,Normal,Normal/images/Normal-9584.png,Normal/masks/Normal-9584.png,True,G19143,train,cc563abfe2383a6090adcdc2935daecb,9161ae16f91efa0a
Normal-9585.png,Normal,Normal/images/Normal-9585.png,Normal/masks/Normal-9585.png,True,G19144,test,558a59fe923d95a217a541a884a2ab10,dc2e435f4b4f0b0a
Normal-9586.png,Normal,Normal/images/Normal-9586.png,Normal/masks/Normal-9586.png,True,G19145,train,65190368157a13bcc03893a07e44adde,dd2ec31b1d4c2e4c
Normal-9587.png,Normal,Normal/images/Normal-9587.png,Normal/masks/Normal-9587.png,True,G19146,train,11a41b8128448631da1e02247926a281,997c8603fb056e78
Normal-9588.png,Normal,Normal/images/Normal-9588.png,Normal/masks/Normal-9588.png,True,G19147,train,47ab86a1987ca98185ca716146729c27,dc3f430f4b0e0b0f
Normal-9589.png,Normal,Normal/images/Normal-9589.png,Normal/masks/Normal-9589.png,True,G19148,train,c37a8d643aac7f9a58dbda75f956753d,d97cc21b4d4a1b4d
Normal-959.png,Normal,Normal/images/Normal-959.png,Normal/masks/Normal-959.png,True,G19149,train,c9797ad7aac24305deea53d49dc23264,812fc25a3ce03fd6
Normal-9590.png,Normal,Normal/images/Normal-9590.png,Normal/masks/Normal-9590.png,True,G19150,train,a4f67cf73e356560fcda587805d64d15,947d42c21bd54b5d
Normal-9591.png,Normal,Normal/images/Normal-9591.png,Normal/masks/Normal-9591.png,True,G19151,train,dae6471d232861bf9b936fcb21791ca0,92682fd1c69c39e6
Normal-9592.png,Normal,Normal/images/Normal-9592.png,Normal/masks/Normal-9592.png,True,G19152,train,e944f3d88907427327ae98c7af8bb9b0,993b861a1d4d19d7
Normal-9593.png,Normal,Normal/images/Normal-9593.png,Normal/masks/Normal-9593.png,True,G19153,val,e3602ccfc2fcb0048c42d812b7f5018f,dd2ec24d4e243fb0
Normal-9594.png,Normal,Normal/images/Normal-9594.png,Normal/masks/Normal-9594.png,True,G19154,train,7f7a72b84df838ce47b9446c11154a4c,dd2ec74b1d4c2c1c
Normal-9595.png,Normal,Normal/images/Normal-9595.png,Normal/masks/Normal-9595.png,True,G19155,val,d1daf4ce2d3c15b27b2fe2ac0fdd7b60,997b861a1f475a19
Normal-9596.png,Normal,Normal/images/Normal-9596.png,Normal/masks/Normal-9596.png,True,G19156,val,36920e9cb8e4c4f044ed2e8c3f061de9,d92d85171ec51f1a
Normal-9597.png,Normal,Normal/images/Normal-9597.png,Normal/masks/Normal-9597.png,True,G19157,test,a6b5cdb818b76b5a731be1b9ff16435c,917806187feca5c7
Normal-9598.png,Normal,Normal/images/Normal-9598.png,Normal/masks/Normal-9598.png,True,G19158,val,525460f3ab951d9ef5834540742960f6,992d071f051f1d4f
Normal-9599.png,Normal,Normal/images/Normal-9599.png,Normal/masks/Normal-9599.png,True,G19159,test,fbe05c08b2e424e1a4f674d23ed5aed4,d128c65a5e4f1b1b
Normal-96.png,Normal,Normal/images/Normal-96.png,Normal/masks/Normal-96.png,True,G19160,train,60894cea1d32bd5fd90a94b28febe338,98ed871ec31e1a2d
Normal-960.png,Normal,Normal/images/Normal-960.png,Normal/masks/Normal-960.png,True,G19161,train,746df0382b43cffb14547f039a775661,d03dc31b9c491ddc
Normal-9600.png,Normal,Normal/images/Normal-9600.png,Normal/masks/Normal-9600.png,True,G19162,train,c7a5ceb0e63e36ef1495cf07c5826121,b86dc61a411f1e4f
Normal-9601.png,Normal,Normal/images/Normal-9601.png,Normal/masks/Normal-9601.png,True,G19163,train,b69192a0adef9e6472fb3b638b811b2f,cc6e53194fbd28a4
Normal-9602.png,Normal,Normal/images/Normal-9602.png,Normal/masks/Normal-9602.png,True,G19164,train,48046f0449a30020684a997fa8d2845c,cd2cc21fd458fce0
Normal-9603.png,Normal,Normal/images/Normal-9603.png,Normal/masks/Normal-9603.png,True,G19165,train,d735c91f02963ebc61027a143e1b0bc1,897807161e5f7978
Normal-9604.png,Normal,Normal/images/Normal-9604.png,Normal/masks/Normal-9604.png,True,G19166,train,34be23780c29e99672afcdbebb4830b0,d129861e1c1f1a7f
Normal-9605.png,Normal,Normal/images/Normal-9605.png,Normal/masks/Normal-9605.png,True,G19167,train,a9c02bea22da78b1683126e020ab55b0,dd2cd0435fc24a1f
Normal-9606.png,Normal,Normal/images/Normal-9606.png,Normal/masks/Normal-9606.png,True,G19168,train,e752ec73603437d9aae2c632a61cef5d,9968871a1d5a5bcd
Normal-9607.png,Normal,Normal/images/Normal-9607.png,Normal/masks/Normal-9607.png,True,G19169,train,0e137529416d0b7cce67b4597c956765,c83d530b4d283d3f
Normal-9608.png,Normal,Normal/images/Normal-9608.png,Normal/masks/Normal-9608.png,True,G19170,train,a9499af4c54a2f066b4f472cc571cafd,997fc71a054b1a8d
Normal-9609.png,Normal,Normal/images/Normal-9609.png,Normal/masks/Normal-9609.png,True,G19171,train,c6b395a74eeb8ce8c5ba055de28ed5ba,817b061e5f59f819
Normal-961.png,Normal,Normal/images/Normal-961.png,Normal/masks/Normal-961.png,True,G19172,train,9b9cda6879273448eb3bbb08e79beaf2,d03fc63b9d4d1cc0
Normal-9610.png,Normal,Normal/images/Normal-9610.png,Normal/masks/Normal-9610.png,True,G19173,train,d68b83e3278d413d1a912f2dc2cc0f2f,dd2ec25a4bc50ba9
Normal-9611.png,Normal,Normal/images/Normal-9611.png,Normal/masks/Normal-9611.png,True,G19174,val,9ab7447af9303f67f05b59aac110e823,897d06161f1b195f
Normal-9612.png,Normal,Normal/images/Normal-9612.png,Normal/masks/Normal-9612.png,True,G19175,test,cf75605b4a9a20f704f011537fc03c7e,817886191d5e797d
Normal-9613.png,Normal,Normal/images/Normal-9613.png,Normal/masks/Normal-9613.png,True,G19176,test,3e7f98e463a9f1dbb333e8758ebea72c,9979871e580d1b6b
Normal-9614.png,Normal,Normal/images/Normal-9614.png,Normal/masks/Normal-9614.png,True,G19177,train,fe83585270584f92cbe5db0adf3564fd,d86cc49253475e57
Normal-9615.png,Normal,Normal/images/Normal-9615.png,Normal/masks/Normal-9615.png,True,G19178,train,a5030729e1bc7a30ed907dc3882a2985,c13bd6095a1b5b1b
Normal-9616.png,Normal,Normal/images/Normal-9616.png,Normal/masks/Normal-9616.png,True,G19179,val,3c14e508b4f8449bab3b34de415328cd,d92b861a192f5ab9
Normal-9617.png,Normal,Normal/images/Normal-9617.png,Normal/masks/Normal-9617.png,True,G19180,train,5d755428497380d2078265bbfb586a77,8139469afbe1e8b1
Normal-9618.png,Normal,Normal/images/Normal-9618.png,Normal/masks/Normal-9618.png,True,G19181,train,60b61fb126a1c38d2bb44ee4aaee6c21,982d861ac55fe95c
Normal-9619.png,Normal,Normal/images/Normal-9619.png,Normal/masks/Normal-9619.png,True,G19182,train,f7b2a6fd28b720ff9d8938ae3daf45fe,887d861e194f5f19
Normal-962.png,Normal,Normal/images/Normal-962.png,Normal/masks/Normal-962.png,True,G19183,train,e0d0983ba81b53d71c27c40116f3871a,903fc61a9d5515c7
Normal-9620.png,Normal,Normal/images/Normal-9620.png,Normal/masks/Normal-9620.png,True,G19184,train,a79ffaf28e02395f50c1e527a6385bdc,c47c4b53094aae7d
Normal-9621.png,Normal,Normal/images/Normal-9621.png,Normal/masks/Normal-9621.png,True,G19185,train,bd914fc133a5ce2dff3396d703ffa19d,807b07161a1f7b1f
Normal-9622.png,Normal,Normal/images/Normal-9622.png,Normal/masks/Normal-9622.png,True,G19186,train,c15edffc4ce8337af5887d54886de403,997b47861e454a1f
Normal-9623.png,Normal,Normal/images/Normal-9623.png,Normal/masks/Normal-9623.png,True,G19187,train,c3e98ec8bebefcc97bb138087b465f46,957807821b5d5d1f
Normal-9624.png,Normal,Normal/images/Normal-9624.png,Normal/masks/Normal-9624.png,True,G19188,train,5ebda8f67d177878ee5e5d48e5b372d3,897904165be9e9f8
Normal-9625.png,Normal,Normal/images/Normal-9625.png,Normal/masks/Normal-9625.png,True,G19189,val,69baa4e924d4f864a4efb67f2ca200cb,803dc3cb1c2d4f3d
Normal-9626.png,Normal,Normal/images/Normal-9626.png,Normal/masks/Normal-9626.png,True,G19190,train,79fb29071efe01a3a81167e6a32231c5,dd2ed20f0d4c1d5c
Normal-9627.png,Normal,Normal/images/Normal-9627.png,Normal/masks/Normal-9627.png,True,G19191,train,79346434bd94f88bfbaa818df1afda8b,8035b23b386d6e6d
Normal-9628.png,Normal,Normal/images/Normal-9628.png,Normal/masks/Normal-9628.png,True,G19192,train,eb7bbb8bea877c64fca45fb523881d80,897987161e1d4f1e
Normal-9629.png,Normal,Normal/images/Normal-9629.png,Normal/masks/Normal-9629.png,True,G19193,val,bc50b1c2e71b9e5b057b8b2b0cac562d,890f061e1b5d5e5e
Normal-963.png,Normal,Normal/images/Normal-963.png,Normal/masks/Normal-963.png,True,G19194,val,14ec08251751646daf5f1f0d3207e42d,d839c25a5e593d38
Normal-9630.png,Normal,Normal/images/Normal-9630.png,Normal/masks/Normal-9630.png,True,G19195,test,6ee365722e1aa26b324d884acabcdc0e,fc2ac71a5d4c594c
Normal-9631.png,Normal,Normal/images/Normal-9631.png,Normal/masks/Normal-9631.png,True,G19196,train,b3e233e1b59d0a8f1a62a02b43f2044c,dc2c438b5d2e6ce8
Normal-9632.png,Normal,Normal/images/Normal-9632.png,Normal/masks/Normal-9632.png,True,G19197,test,b3bd88d8a63b171b167a0c333c32df3d,d81dd60a1c0e5f3b
Normal-9633.png,Normal,Normal/images/Normal-9633.png,Normal/masks/Normal-9633.png,True,G19198,train,1bbe869c1d836ff9586ad33007cb204c,917d471a4b4b0f4b
Normal-9634.png,Normal,Normal/images/Normal-9634.png,Normal/masks/Normal-9634.png,True,G19199,test,4a1c736d30c5b00c600d550239703a0e,8939c61b1c6d1b6d
Normal-9635.png,Normal,Normal/images/Normal-9635.png,Normal/masks/Normal-9635.png,True,G19200,train,aaefc98a409c41fc217988a081aff253,a87d84561acd5a5e
Normal-9636.png,Normal,Normal/images/Normal-9636.png,Normal/masks/Normal-9636.png,True,G19201,train,645b7fd71a750e864978c52b10a4aa32,9969861e1acf7a98
Normal-9637.png,Normal,Normal/images/Normal-9637.png,Normal/masks/Normal-9637.png,True,G19202,train,9f7c442765d49f8530b28f676e0f86b7,817bc7165e471e1c
Normal-9638.png,Normal,Normal/images/Normal-9638.png,Normal/masks/Normal-9638.png,True,G19203,train,1698b332a2b05ee0247c561d0f7cbd11,897d86961a879759
Normal-9639.png,Normal,Normal/images/Normal-9639.png,Normal/masks/Normal-9639.png,True,G19204,train,be67736e20e656877206fba305efc1f1,dd2a521f0a4b0fae
Normal-964.png,Normal,Normal/images/Normal-964.png,Normal/masks/Normal-964.png,True,G19205,train,be82857edf598018cfec6f86c14ca672,d03dc23d9c6938e9
Normal-9640.png,Normal,Normal/images/Normal-9640.png,Normal/masks/Normal-9640.png,True,G19206,train,518656365ef31ee0db6d904ee63104eb,dc3fc24f0d420f3c
Normal-9641.png,Normal,Normal/images/Normal-9641.png,Normal/masks/Normal-9641.png,True,G19207,train,6f7bcc23b50aa8c7f3ce71f00a4c0476,c83f461f544f184f
Normal-9642.png,Normal,Normal/images/Normal-9642.png,Normal/masks/Normal-9642.png,True,G19208,train,2aa54be9818be6957cc902a446aed9b2,c92f474e094f0e1f
Normal-9643.png,Normal,Normal/images/Normal-9643.png,Normal/masks/Normal-9643.png,True,G19209,val,a54c3b1cd86dac379d989f3379ee70cb,dd2dc21f1cc61d1c
Normal-9644.png,Normal,Normal/images/Normal-9644.png,Normal/masks/Normal-9644.png,True,G19210,train,02143963fb8b0cbaac21e2665a5021ba,913a1618795a7b6b
Normal-9645.png,Normal,Normal/images/Normal-9645.png,Normal/masks/Normal-9645.png,True,G19211,train,f7fc81acce073d88bbcf44b6bc7a7e36,916986961a1f575b
Normal-9646.png,Normal,Normal/images/Normal-9646.png,Normal/masks/Normal-9646.png,True,G19212,train,2dd1e35f7eed29573fe012a70e737b06,cd3d520f420f5f0d
Normal-9647.png,Normal,Normal/images/Normal-9647.png,Normal/masks/Normal-9647.png,True,G19213,val,a728118f4a5843791d20973d440adbe6,d1790e06d01f5f4b
Normal-9648.png,Normal,Normal/images/Normal-9648.png,Normal/masks/Normal-9648.png,True,G19214,val,9fe3e4f9109bff8ea5ea051e6fb7f80c,887d861e185d5b79
Normal-9649.png,Normal,Normal/images/Normal-9649.png,Normal/masks/Normal-9649.png,True,G19215,train,93457d7a7cf03c7876a4f9abd6030aea,817e07121a5f595f
Normal-965.png,Normal,Normal/images/Normal-965.png,Normal/masks/Normal-965.png,True,G19216,train,c53e974a707cd96f7f9a4a4ffe2383bb,d02bd25e5e5b1c49
Normal-9650.png,Normal,Normal/images/Normal-9650.png,Normal/masks/Normal-9650.png,True,G19217,test,574b7292a5bce99d3c181a883984ec11,897f061a49e5b9e8
Normal-9651.png,Normal,Normal/images/Normal-9651.png,Normal/masks/Normal-9651.png,True,G19218,test,dc5766f8942377642fd644ec0bdf8022,dd3cc01bd54a1f45
Normal-9652.png,Normal,Normal/images/Normal-9652.png,Normal/masks/Normal-9652.png,True,G19219,test,3dce156698dafed6876591cf694e5ccc,917ad71e5909581f
Normal-9653.png,Normal,Normal/images/Normal-9653.png,Normal/masks/Normal-9653.png,True,G19220,train,877ec57ef1a16dccf3af1b4fc683e9f7,d93dc21b020f39bd
Normal-9654.png,Normal,Normal/images/Normal-9654.png,Normal/masks/Normal-9654.png,True,G19221,train,75683f74d954d6bf9143bb30a75393b8,817b061e585f68f9
Normal-9655.png,Normal,Normal/images/Normal-9655.png,Normal/masks/Normal-9655.png,True,G19222,train,85ad64784b6ca29ac870a249d08b2833,f96dd61b45161945
Normal-9656.png,Normal,Normal/images/Normal-9656.png,Normal/masks/Normal-9656.png,True,G19223,train,4c850dad06ef0f056c30e97755e73ef1,9969c71a174f1a59
Normal-9657.png,Normal,Normal/images/Normal-9657.png,Normal/masks/Normal-9657.png,True,G19224,train,05547af57d2a73dc05482c2a07984e8e,986c461b1b4d7c3d
Normal-9658.png,Normal,Normal/images/Normal-9658.png,Normal/masks/Normal-9658.png,True,G19225,train,e96c7622c3034c3d23268ee0afa51e4b,997cc65bc54f1a06
Normal-9659.png,Normal,Normal/images/Normal-9659.png,Normal/masks/Normal-9659.png,True,G19226,train,b1e6a8f9cb0f795f7a5371f1a702c3b9,cc2d82171e5e3d59
Normal-966.png,Normal,Normal/images/Normal-966.png,Normal/masks/Normal-966.png,True,G19227,train,6d3066aef35f5c1cfa78c4209075ada4,d12cc63afc6838ec
Normal-9660.png,Normal,Normal/images/Normal-9660.png,Normal/masks/Normal-9660.png,True,G19228,train,9db0663c4a49653cbbeec96c8366f34f,997bc79e18413ad8
Normal-9661.png,Normal,Normal/images/Normal-9661.png,Normal/masks/Normal-9661.png,True,G19229,train,f10a067ec332e240c20a18dcf742ceda,d57f420b5a0d0b3d
Normal-9662.png,Normal,Normal/images/Normal-9662.png,Normal/masks/Normal-9662.png,True,G19230,train,dde8c01f41e747d34d550763b4cf23a6,d93bc41e515e1a69
Normal-9663.png,Normal,Normal/images/Normal-9663.png,Normal/masks/Normal-9663.png,True,G19231,train,8f14e8b853d6501c66f4a656128b3499,99691e871a5edc58
Normal-9664.png,Normal,Normal/images/Normal-9664.png,Normal/masks/Normal-9664.png,True,G19232,val,13283200f9ceadf461f9dc64ce916b84,997f06184939f9e4
Normal-9665.png,Normal,Normal/images/Normal-9665.png,Normal/masks/Normal-9665.png,True,G19233,val,c504c27180704be8f64375f13b645b23,dd08461e1a686f7d
Normal-9666.png,Normal,Normal/images/Normal-9666.png,Normal/masks/Normal-9666.png,True,G19234,val,cb7092795801e1c8b55b59ba8daa7bcd,d02957d61e9c4b1d
Normal-9667.png,Normal,Normal/images/Normal-9667.png,Normal/masks/Normal-9667.png,True,G19235,test,0569841c01de220f20c0776becd54e67,9961ce1a9e29bab8
Normal-9668.png,Normal,Normal/images/Normal-9668.png,Normal/masks/Normal-9668.png,True,G19236,test,d8524da3e6da55e7a6100699c331eb8e,8828c51bd55f1e5d
Normal-9669.png,Normal,Normal/images/Normal-9669.png,Normal/masks/Normal-9669.png,True,G19237,test,c3e686db2c2e0e49850b0320bb847a09,dc3cc20b1e4f1d6c
Normal-967.png,Normal,Normal/images/Normal-967.png,Normal/masks/Normal-967.png,True,G19238,train,6f6b39603eb1f9e502960ac57df0f750,903ac71ede513935
Normal-9670.png,Normal,Normal/images/Normal-9670.png,Normal/masks/Normal-9670.png,True,G19239,train,c7e974e7556596356dd7c645df04227a,9939c4161f4d5d5a
Normal-9671.png,Normal,Normal/images/Normal-9671.png,Normal/masks/Normal-9671.png,True,G19240,train,1ef28394d71b615d1db628d7569c9ffc,dc2c52435f0e2d1f
Normal-9672.png,Normal,Normal/images/Normal-9672.png,Normal/masks/Normal-9672.png,True,G19241,train,5bb5a5445861286b598220813d45242e,d03dd35b1d086d39
Normal-9673.png,Normal,Normal/images/Normal-9673.png,Normal/masks/Normal-9673.png,True,G19242,train,5d7156d175d6ffe1fe392b201db5ed9f,c858d65f4d0e0f87
Normal-9674.png,Normal,Normal/images/Normal-9674.png,Normal/masks/Normal-9674.png,True,G19243,train,83cfa133eb4da302604246f731b44235,cc2e535b1b5d2a38
Normal-9675.png,Normal,Normal/images/Normal-9675.png,Normal/masks/Normal-9675.png,True,G19244,train,b1feb88b980ba40aae3846a5321d1b5a,8b291e1a5bb978a9
Normal-9676.png,Normal,Normal/images/Normal-9676.png,Normal/masks/Normal-9676.png,True,G19245,train,779587f2d7d6273c2785fbdf56c7c089,dc2d42c31b0d6e3b
Normal-9677.png,Normal,Normal/images/Normal-9677.png,Normal/masks/Normal-9677.png,True,G19246,train,a76913c62098bf3f9190fe1cf2c8309e,d947165b493aacb8
Normal-9678.png,Normal,Normal/images/Normal-9678.png,Normal/masks/Normal-9678.png,True,G19247,train,60407e59055dce76bf8fae583be9da55,9178861e194f4f79
Normal-9679.png,Normal,Normal/images/Normal-9679.png,Normal/masks/Normal-9679.png,True,G19248,val,bb6431d9325c78c004627d9f3d3b8a56,dd2f42931b424f5a
Normal-968.png,Normal,Normal/images/Normal-968.png,Normal/masks/Normal-968.png,True,G19249,train,af73fff083e02e82eb19aaefb78bf95a,917e863a8d2f3a86
Normal-9680.png,Normal,Normal/images/Normal-9680.png,Normal/masks/Normal-9680.png,True,G19250,train,23792013b428dc75b13b6b245ebf69c7,cc3dc2411f474e3d
Normal-9681.png,Normal,Normal/images/Normal-9681.png,Normal/masks/Normal-9681.png,True,G19251,train,4f7ac34c4a0b47592ca732b835a56394,d93d47065b454e5a
Normal-9682.png,Normal,Normal/images/Normal-9682.png,Normal/masks/Normal-9682.png,True,G19252,test,d2ae44bfb6e2b917784ce63369a1bdfa,817bd75e5a187819
Normal-9683.png,Normal,Normal/images/Normal-9683.png,Normal/masks/Normal-9683.png,True,G19253,train,aef3b2669c915a3d2b662c6507221d6b,8178461f1b6f7938
Normal-9684.png,Normal,Normal/images/Normal-9684.png,Normal/masks/Normal-9684.png,True,G19254,val,7508b61b63a0bca6101cf4ef35d95e1c,f979c5161ac74718
Normal-9685.png,Normal,Normal/images/Normal-9685.png,Normal/masks/Normal-9685.png,True,G19255,test,4c300c9d3396d82a8baa81d52971e003,dd2a53171f092c3e
Normal-9686.png,Normal,Normal/images/Normal-9686.png,Normal/masks/Normal-9686.png,True,G19256,val,6183042065263eb0e00ff150fc63d643,d97806561b184bfb
Normal-9687.png,Normal,Normal/images/Normal-9687.png,Normal/masks/Normal-9687.png,True,G19257,train,d362b39241709b0b27f581de469413c5,d93fc6c31b554a09
Normal-9688.png,Normal,Normal/images/Normal-9688.png,Normal/masks/Normal-9688.png,True,G19258,test,feaaa72998d3d223b5572f47d579b49a,dd2e461b4a4d3cca
Normal-9689.png,Normal,Normal/images/Normal-9689.png,Normal/masks/Normal-9689.png,True,G19259,train,46c05734110566490218fe7e41f1a376,d979461a5e4f1a49
Normal-969.png,Normal,Normal/images/Normal-969.png,Normal/masks/Normal-969.png,True,G19260,val,6b53acb16cba8294839e50e74874806d,dd28d20bdf681c78
Normal-9690.png,Normal,Normal/images/Normal-9690.png,Normal/masks/Normal-9690.png,True,G19261,train,2d961112eeb88f93faefb36e10e624ca,897bd64a185d4b39
Normal-9691.png,Normal,Normal/images/Normal-9691.png,Normal/masks/Normal-9691.png,True,G19262,train,1fa3cc3b6592ea7dd09036006eb78174,d43d4b43911a8f6e
Normal-9692.png,Normal,Normal/images/Normal-9692.png,Normal/masks/Normal-9692.png,True,G19263,train,1a730980fd7e1d1e7dfe2a5d82a82c46,81785616185faf3b
Normal-9693.png,Normal,Normal/images/Normal-9693.png,Normal/masks/Normal-9693.png,True,G19264,train,06fb40fa6d5a700ce51be4c1b4ca5581,dd2856c35f114f4a
Normal-9694.png,Normal,Normal/images/Normal-9694.png,Normal/masks/Normal-9694.png,True,G19265,train,d39107b85f09999984c67810e2883eb3,dd7cd21bd40b190b
Normal-9695.png,Normal,Normal/images/Normal-9695.png,Normal/masks/Normal-9695.png,True,G19266,train,c8baed19812ec36013eafe4d3794b868,993d051e1a4d4f7a
Normal-9696.png,Normal,Normal/images/Normal-9696.png,Normal/masks/Normal-9696.png,True,G19267,train,4445970e753e904b06e46b42ddb95467,817b96864b1d1e5b
Normal-9697.png,Normal,Normal/images/Normal-9697.png,Normal/masks/Normal-9697.png,True,G19268,train,cfdc9844c9b597fff109041fd139fa6d,d9af061e071a551f
Normal-9698.png,Normal,Normal/images/Normal-9698.png,Normal/masks/Normal-9698.png,True,G19269,train,b87b7e44836de06e18b369d294375909,983d471f454f1d45
Normal-9699.png,Normal,Normal/images/Normal-9699.png,Normal/masks/Normal-9699.png,True,G19270,train,79073a521262f01da9aa4b6d1d322bb8,d96dc65a4d585949
Normal-97.png,Normal,Normal/images/Normal-97.png,Normal/masks/Normal-97.png,True,G19271,train,87ef115adde4fa36c50ba13a4ec753ad,d47e871f8d0c68c6
Normal-970.png,Normal,Normal/images/Normal-970.png,Normal/masks/Normal-970.png,True,G19272,val,096b9b15a75393401f0765df4ab32f52,947c831bc93d789c
Normal-9700.png,Normal,Normal/images/Normal-9700.png,Normal/masks/Normal-9700.png,True,G19273,test,0dd9637a406a3dc4da24bfa4360ea3a3,dd2ec2c35d4e491c
Normal-9701.png,Normal,Normal/images/Normal-9701.png,Normal/masks/Normal-9701.png,True,G19274,train,703e42f433352b39442fe02b3ba08b46,997c065a79e9b8a1
Normal-9702.png,Normal,Normal/images/Normal-9702.png,Normal/masks/Normal-9702.png,True,G19275,train,12f5833211df6a73a170abe0ffe20750,d42f02c33ff435d0
Normal-9703.png,Normal,Normal/images/Normal-9703.png,Normal/masks/Normal-9703.png,True,G19276,train,0005fd114eb9934999952383375819c0,816847961a5d6f7a
Normal-9704.png,Normal,Normal/images/Normal-9704.png,Normal/masks/Normal-9704.png,True,G19277,train,6ada7be60da54f7225ba3869114dbb53,816806185f5f797b
Normal-9705.png,Normal,Normal/images/Normal-9705.png,Normal/masks/Normal-9705.png,True,G19278,val,077fc0985168b82fc6a56067c231fccd,8d7d871e5a8319a9
Normal-9706.png,Normal,Normal/images/Normal-9706.png,Normal/masks/Normal-9706.png,True,G19279,test,06cf2730fabc8fa2c4b470ae6f3a4028,815f5e381b39a9b1
Normal-9707.png,Normal,Normal/images/Normal-9707.png,Normal/masks/Normal-9707.png,True,G19280,test,8416afbf4335d3ce74b715fb32b2e35e,992dc61b494f1c5d
Normal-9708.png,Normal,Normal/images/Normal-9708.png,Normal/masks/Normal-9708.png,True,G19281,test,4454ca492f35c167358c33c8943b7359,dd06961f451b4c3e
Normal-9709.png,Normal,Normal/images/Normal-9709.png,Normal/masks/Normal-9709.png,True,G19282,train,398bce8752f09e86c8b58c41de7e2679,dc6ac31a4e561b4b
Normal-971.png,Normal,Normal/images/Normal-971.png,Normal/masks/Normal-971.png,True,G19283,train,e0e249bfadecb0d84d7a875304e03533,813ec63aed4b2cc5
Normal-9710.png,Normal,Normal/images/Normal-9710.png,Normal/masks/Normal-9710.png,True,G19284,train,5410d3674a80f3ad744f07153b06c1dc,91694e165a1ecf3a
Normal-9711.png,Normal,Normal/images/Normal-9711.png,Normal/masks/Normal-9711.png,True,G19285,train,9b5e1c63704b58eceb35c2e4bb1d07c0,983fc64a1d6c68bc
Normal-9712.png,Normal,Normal/images/Normal-9712.png,Normal/masks/Normal-9712.png,True,G19286,train,64b687e0d9e89f034a6f7c6384eeb025,c13bd61a789a5b38
Normal-9713.png,Normal,Normal/images/Normal-9713.png,Normal/masks/Normal-9713.png,True,G19287,train,742c20ab29031cda417e48ef091ef03f,d5380bd3d8f286a3
Normal-9714.png,Normal,Normal/images/Normal-9714.png,Normal/masks/Normal-9714.png,True,G19288,train,097028875c547e7ebb94742ef2d67ec0,c568ce1d19afa49a
Normal-9715.png,Normal,Normal/images/Normal-9715.png,Normal/masks/Normal-9715.png,True,G19289,test,02c228d7659aa37cae05eca050cf544a,99291e1f3a4d6939
Normal-9716.png,Normal,Normal/images/Normal-9716.png,Normal/masks/Normal-9716.png,True,G19290,train,7c7d8c7c5996bc192b6b0a5ef5ab9f51,d13d86101f4f1d5b
Normal-9717.png,Normal,Normal/images/Normal-9717.png,Normal/masks/Normal-9717.png,True,G19291,train,a4b3d11193242e670f5ac8d24af91f13,817a9618195f393f
Normal-9718.png,Normal,Normal/images/Normal-9718.png,Normal/masks/Normal-9718.png,True,G19292,train,5f289ef39c7e92c7f8b3accf0796c55d,993d865a18dc5b39
Normal-9719.png,Normal,Normal/images/Normal-9719.png,Normal/masks/Normal-9719.png,True,G19293,val,bed27962de078a30048d54f6c5fb62d0,f975961655515945
Normal-972.png,Normal,Normal/images/Normal-972.png,Normal/masks/Normal-972.png,True,G19294,train,c04791cdbe5f7b4eb888c121fa05e375,d47ec71f8d4c0cc4
Normal-9720.png,Normal,Normal/images/Normal-9720.png,Normal/masks/Normal-9720.png,True,G19295,val,440457e45f02a3abd3c3c9d0cf8028f5,9879871a055e1f4f
Normal-9721.png,Normal,Normal/images/Normal-9721.png,Normal/masks/Normal-9721.png,True,G19296,test,d5017a8d8b618f2f5a3a131a07072a5c,d47d434b0e6f2d0c
Normal-9722.png,Normal,Normal/images/Normal-9722.png,Normal/masks/Normal-9722.png,True,G19297,test,f17d66313965c21a5a6841114c0370e7,dd3d06161a0e6d7a
Normal-9723.png,Normal,Normal/images/Normal-9723.png,Normal/masks/Normal-9723.png,True,G19298,train,30b4540e2f03b879fc4b8b862664a7e0,9c6c461b0b4f1bb9
Normal-9724.png,Normal,Normal/images/Normal-9724.png,Normal/masks/Normal-9724.png,True,G19299,test,ea0f5e989e28b254721ac30ba6b3e50e,c129bf070e161e5f
Normal-9725.png,Normal,Normal/images/Normal-9725.png,Normal/masks/Normal-9725.png,True,G19300,train,5dcc3c2d5f6ff8baccffc65e3018fe39,d979071e1f165a15
Normal-9726.png,Normal,Normal/images/Normal-9726.png,Normal/masks/Normal-9726.png,True,G19301,val,3515c66f5a81f30c032813aff735442d,dd2e065f0f484d6a
Normal-9727.png,Normal,Normal/images/Normal-9727.png,Normal/masks/Normal-9727.png,True,G19302,train,3b1c4289ca5f621f5983c453fad6a660,dd2fc64b0d4c4a69
Normal-9728.png,Normal,Normal/images/Normal-9728.png,Normal/masks/Normal-9728.png,True,G19303,train,1fb5ad6ddebb6d3666085d19aa0ecc37,8179161a19ff6939
Normal-9729.png,Normal,Normal/images/Normal-9729.png,Normal/masks/Normal-9729.png,True,G19304,train,31e026a1e3391d29f91df45a983cd371,993d871b151f3945
Normal-973.png,Normal,Normal/images/Normal-973.png,Normal/masks/Normal-973.png,True,G19305,train,5dfb0d56e2eb75016496c29c6fda7a6d,9139c71e9ec80bb9
Normal-9730.png,Normal,Normal/images/Normal-9730.png,Normal/masks/Normal-9730.png,True,G19306,train,5e125e249bd8019e437375034956d425,dc2ec25b1d446f1c
Normal-9731.png,Normal,Normal/images/Normal-9731.png,Normal/masks/Normal-9731.png,True,G19307,train,321a641cbb6806313ace22d19c862358,dd2cd21b440b5d4f
Normal-9732.png,Normal,Normal/images/Normal-9732.png,Normal/masks/Normal-9732.png,True,G19308,train,1fb717b0a1f75b1756085a37eae32ffa,99698618155d3adf
Normal-9733.png,Normal,Normal/images/Normal-9733.png,Normal/masks/Normal-9733.png,True,G19309,train,846fbb3c5fb2d887ae00796c5ea01505,8879c71a1d4f3b91
Normal-9734.png,Normal,Normal/images/Normal-9734.png,Normal/masks/Normal-9734.png,True,G19310,train,d6b8e638199b8b3fc4912c4823593952,d979861a194d3b69
Normal-9735.png,Normal,Normal/images/Normal-9735.png,Normal/masks/Normal-9735.png,True,G19311,train,6493165785faaced35ec1a66b0110488,cc7d430b0f2c3cad
Normal-9736.png,Normal,Normal/images/Normal-9736.png,Normal/masks/Normal-9736.png,True,G19312,train,e2b2b95f7214d81180808e1daf6aa0b0,c93b56de5a104b2d
Normal-9737.png,Normal,Normal/images/Normal-9737.png,Normal/masks/Normal-9737.png,True,G19313,train,d4efafbd9ee044fd9568ca539f32b1b2,cc3d420f0c2f3dbc
Normal-9738.png,Normal,Normal/images/Normal-9738.png,Normal/masks/Normal-9738.png,True,G19314,val,9adc6bd94d29198f236ee8d96893cabe,812e06045f5e7e7d
Normal-9739.png,Normal,Normal/images/Normal-9739.png,Normal/masks/Normal-9739.png,True,G19315,train,a1953dad7834e1b5be192fe4af3a7a9a,dc28831f182f3cbd
Normal-974.png,Normal,Normal/images/Normal-974.png,Normal/masks/Normal-974.png,True,G19316,val,c55a1e0025c3aee1c26b70ae4d48d1f3,d4e4c30fe52ce44e
Normal-9740.png,Normal,Normal/images/Normal-9740.png,Normal/masks/Normal-9740.png,True,G19317,val,1d01540aa76ca7e190f2a50e0d02cac6,8978c7961b85171f
Normal-9741.png,Normal,Normal/images/Normal-9741.png,Normal/masks/Normal-9741.png,True,G19318,test,d0d4d871035a5723a2ba61e02bb29ff2,997d071a4d6938e9
Normal-9742.png,Normal,Normal/images/Normal-9742.png,Normal/masks/Normal-9742.png,True,G19319,test,51fb38ddcc669baf608c7212f1afddec,dc7ad30b4d0b1e0b
Normal-9743.png,Normal,Normal/images/Normal-9743.png,Normal/masks/Normal-9743.png,True,G19320,train,67f1ebea815972c09a516e848b8d71b5,8dbcd7c3f209090f
Normal-9744.png,Normal,Normal/images/Normal-9744.png,Normal/masks/Normal-9744.png,True,G19321,train,05285c6e1e2323c64261c5fdc8a3c84b,997a87161e875e38
Normal-9745.png,Normal,Normal/images/Normal-9745.png,Normal/masks/Normal-9745.png,True,G19322,train,c19668a9272bcee4a90e070ef140f9c6,993b861a196d7eb0
Normal-9746.png,Normal,Normal/images/Normal-9746.png,Normal/masks/Normal-9746.png,True,G19323,train,d0d5241ac34751bef5f85ba8bcc8a3a8,99ea861f851a1d4f
Normal-9747.png,Normal,Normal/images/Normal-9747.png,Normal/masks/Normal-9747.png,True,G19324,train,4a009126bd532c126e172133f9b8db8c,d52f02d74b450f0f
Normal-9748.png,Normal,Normal/images/Normal-9748.png,Normal/masks/Normal-9748.png,True,G19325,train,176db74aad73dd9f3d40d77f4f017706,817907861e1f4dde
Normal-9749.png,Normal,Normal/images/Normal-9749.png,Normal/masks/Normal-9749.png,True,G19326,test,35568c060646c9968f2c578d7634470a,dd3e421a4568b8ed
Normal-975.png,Normal,Normal/images/Normal-975.png,Normal/masks/Normal-975.png,True,G19327,train,c70a71f356fa24044e043551bf4c657c,d029c20abfe93cf4
Normal-9750.png,Normal,Normal/images/Normal-9750.png,Normal/masks/Normal-9750.png,True,G19328,train,bfd0884e4e2bab0025c392c8308ee485,c93c0e07197d6e78
Normal-9751.png,Normal,Normal/images/Normal-9751.png,Normal/masks/Normal-9751.png,True,G19329,train,50652b00ce57a264a730e20bfded718b,9930861f974cbae9
Normal-9752.png,Normal,Normal/images/Normal-9752.png,Normal/masks/Normal-9752.png,True,G19330,train,fa4a7a610f84a5fb26d30d722be80f6e,8919165e1a5b7b3a
Normal-9753.png,Normal,Normal/images/Normal-9753.png,Normal/masks/Normal-9753.png,True,G19331,train,b2412d31056e36c100480eedbf5ac7b9,917c065a1a4e7b3d
Normal-9754.png,Normal,Normal/images/Normal-9754.png,Normal/masks/Normal-9754.png,True,G19332,train,7ec4f8119fe75f3276b5a811197058ba,d12fc61215c71e5b
Normal-9755.png,Normal,Normal/images/Normal-9755.png,Normal/masks/Normal-9755.png,True,G19333,val,0b4bfee16c9062479048fce400d419d5,992e865a19457a7d
Normal-9756.png,Normal,Normal/images/Normal-9756.png,Normal/masks/Normal-9756.png,True,G19334,val,29aa4014e7b15d7741a06d1bd7d164ea,d86f051613475e3e
Normal-9757.png,Normal,Normal/images/Normal-9757.png,Normal/masks/Normal-9757.png,True,G19335,test,b5ac5ac2c578015d1a15890a56303039,8168965b1b7c395d
Normal-9758.png,Normal,Normal/images/Normal-9758.png,Normal/masks/Normal-9758.png,True,G19336,train,4e39c273e1f53e5985b7b0f5ab3d115f,916816185ffa78e5
Normal-9759.png,Normal,Normal/images/Normal-9759.png,Normal/masks/Normal-9759.png,True,G19337,val,4da6d897329e2e6f13c9040c01b5d670,dc3dc25b1bc60b19
Normal-976.png,Normal,Normal/images/Normal-976.png,Normal/masks/Normal-976.png,True,G19338,train,62aa119bd26b704ce3b3c5c55f614f80,802bc65edd5115d7
Normal-9760.png,Normal,Normal/images/Normal-9760.png,Normal/masks/Normal-9760.png,True,G19339,train,d73c574924e235cb1bd339cacd518421,dd2c525b550e1d4d
Normal-9761.png,Normal,Normal/images/Normal-9761.png,Normal/masks/Normal-9761.png,True,G19340,train,bf6fb93ad3711916af7a5837cf03e751,cd2e50430b1d6e6f
Normal-9762.png,Normal,Normal/images/Normal-9762.png,Normal/masks/Normal-9762.png,True,G19341,train,8d13a2a98744481554093a2cb84cc962,8578470b7b0fb839
Normal-9763.png,Normal,Normal/images/Normal-9763.png,Normal/masks/Normal-9763.png,True,G19342,test,4761363da04dd5e417ccfcc30ac8d8e2,c43d83431f1c4b5f
Normal-9764.png,Normal,Normal/images/Normal-9764.png,Normal/masks/Normal-9764.png,True,G19343,train,dddd469f96780bcc22e51c7c79f569c1,887f071a1d293bb5
Normal-9765.png,Normal,Normal/images/Normal-9765.png,Normal/masks/Normal-9765.png,True,G19344,train,f91af9f7e5c205fc605a0d62b956cd1e,985f475b29653472
Normal-9766.png,Normal,Normal/images/Normal-9766.png,Normal/masks/Normal-9766.png,True,G19345,train,ec7e7729142dfc55c43ff3ff0f104548,986f871a175d1a4b
Normal-9767.png,Normal,Normal/images/Normal-9767.png,Normal/masks/Normal-9767.png,True,G19346,train,70f1b86dd0b3e9d284a2fce08ccfd217,897dc71a834f198b
Normal-9768.png,Normal,Normal/images/Normal-9768.png,Normal/masks/Normal-9768.png,True,G19347,train,01aa6a7745afba9da9609dc567e30f84,d939c6561a155e0f
Normal-9769.png,Normal,Normal/images/Normal-9769.png,Normal/masks/Normal-9769.png,True,G19348,train,edbde228e033e91a47d141f11412a4a6,dd0e471b0c6d29bc
Normal-977.png,Normal,Normal/images/Normal-977.png,Normal/masks/Normal-977.png,True,G19349,train,df7f74cc65c6afcdd99d220e31c40dc2,9046c75addcd29e1
Normal-9770.png,Normal,Normal/images/Normal-9770.png,Normal/masks/Normal-9770.png,True,G19350,train,fb4356b5aef6141098d29b83698c0e9b,dc2cc30b8c4e0ffc
Normal-9771.png,Normal,Normal/images/Normal-9771.png,Normal/masks/Normal-9771.png,True,G19351,val,ab4200d769576ecc72a0c85ccbd0c777,dd3cc21f96431b49
Normal-9772.png,Normal,Normal/images/Normal-9772.png,Normal/masks/Normal-9772.png,True,G19352,train,35be0cb9ed6dc1d2d7924e8c2db8d55f,dc3dd30d1c4d1c4d
Normal-9773.png,Normal,Normal/images/Normal-9773.png,Normal/masks/Normal-9773.png,True,G19353,train,867cac8ed0104295a7bbd68a4888822c,cd2ec25b410b1b6f
Normal-9774.png,Normal,Normal/images/Normal-9774.png,Normal/masks/Normal-9774.png,True,G19354,test,e2c6370405c76cb878c00ce9a428010e,dcbe0341161d2f5d
Normal-9775.png,Normal,Normal/images/Normal-9775.png,Normal/masks/Normal-9775.png,True,G19355,train,6a0f28717dd1a640ac122852f2a663e6,d93986161e565a5d
Normal-9776.png,Normal,Normal/images/Normal-9776.png,Normal/masks/Normal-9776.png,True,G19356,val,c915d713746dbe925fd2acb5ca998a1b,91fbc614145f1d55
Normal-9777.png,Normal,Normal/images/Normal-9777.png,Normal/masks/Normal-9777.png,True,G19357,val,fc5f18391e773d7e56476474296f7753,817f06165b6de938
Normal-9778.png,Normal,Normal/images/Normal-9778.png,Normal/masks/Normal-9778.png,True,G19358,test,94ca46a2ec5a23656baa0d5e7cad36cf,9979c7061a194f5b
Normal-9779.png,Normal,Normal/images/Normal-9779.png,Normal/masks/Normal-9779.png,True,G19359,test,7c74c79b221fbd8d5e2cab562a988b31,8068865f1f4f5c3d
Normal-978.png,Normal,Normal/images/Normal-978.png,Normal/masks/Normal-978.png,True,G19360,train,1a7ad9a005ea2f7713910a4e7c1d5d01,d17faf3a09092c4b
Normal-9780.png,Normal,Normal/images/Normal-9780.png,Normal/masks/Normal-9780.png,True,G19361,train,fb2181c1520347e7f83af338b2946731,d42d470f4e4b4b4b
Normal-9781.png,Normal,Normal/images/Normal-9781.png,Normal/masks/Normal-9781.png,True,G19362,train,9dc501a9f3c83cc94eaef70c3ec02f26,d979c51e94461e3a
Normal-9782.png,Normal,Normal/images/Normal-9782.png,Normal/masks/Normal-9782.png,True,G19363,train,30b43f636c710473d94cc7055f07587c,d46f53554a0b5b0b
Normal-9783.png,Normal,Normal/images/Normal-9783.png,Normal/masks/Normal-9783.png,True,G19364,train,f1582402774a67c3f8e98247ae8c0396,dc2fc3430f8a0dae
Normal-9784.png,Normal,Normal/images/Normal-9784.png,Normal/masks/Normal-9784.png,True,G19365,train,f9d81204aede981dc68803a095da1aeb,d92c821b5d68bde4
Normal-9785.png,Normal,Normal/images/Normal-9785.png,Normal/masks/Normal-9785.png,True,G19366,train,dcd2a2586508b9ca856ccb7d6ce8180b,dc4c434b4f0e2b2f
Normal-9786.png,Normal,Normal/images/Normal-9786.png,Normal/masks/Normal-9786.png,True,G19367,train,e0ceae4f2bb012279a3dc8bd8dacd72a,8979c616fa4aec1a
Normal-9787.png,Normal,Normal/images/Normal-9787.png,Normal/masks/Normal-9787.png,True,G19368,train,337514366e320ef1a2ff4431d137f232,917b44161f4b7b38
Normal-9788.png,Normal,Normal/images/Normal-9788.png,Normal/masks/Normal-9788.png,True,G19369,train,3e2ac40225c6e6b9b8af1f345d1fd153,81788e56b81fee1a
Normal-9789.png,Normal,Normal/images/Normal-9789.png,Normal/masks/Normal-9789.png,True,G19370,train,63aef6595b65f61b89e638f26782cf02,856e074a0fec6b69
Normal-979.png,Normal,Normal/images/Normal-979.png,Normal/masks/Normal-979.png,True,G19371,test,b002ba454638a4e6580650c70de89982,d56fa70a4d4d6c44
Normal-9790.png,Normal,Normal/images/Normal-9790.png,Normal/masks/Normal-9790.png,True,G19372,train,24c7bc51d5e7014662e8e475fc552614,d43d43435d4f451d
Normal-9791.png,Normal,Normal/images/Normal-9791.png,Normal/masks/Normal-9791.png,True,G19373,train,e08b70c80b748b150e9c83d52fcaaee0,f97f961e081e180f
Normal-9792.png,Normal,Normal/images/Normal-9792.png,Normal/masks/Normal-9792.png,True,G19374,train,45b49c67fb616ecb859be1890b42b16b,dc7d430b0f4a2f0e
Normal-9793.png,Normal,Normal/images/Normal-9793.png,Normal/masks/Normal-9793.png,True,G19375,val,9f8aef0a2bf1f2c221a0e1584752018f,d43d824b5f544f0d
Normal-9794.png,Normal,Normal/images/Normal-9794.png,Normal/masks/Normal-9794.png,True,G19376,val,823edecd7a71ba2625ff3c4b6e12c0c4,8168161e5b5f787a
Normal-9795.png,Normal,Normal/images/Normal-9795.png,Normal/masks/Normal-9795.png,True,G19377,test,c4860a544fbd72bcb3e3154f4377cda2,816a16195d1e5d5f
Normal-9796.png,Normal,Normal/images/Normal-9796.png,Normal/masks/Normal-9796.png,True,G19378,val,8a78e47022ca34db30a1763e681c9710,997d861a1f5838cb
Normal-9797.png,Normal,Normal/images/Normal-9797.png,Normal/masks/Normal-9797.png,True,G19379,test,b098439f969705d81e2327c91ac57466,987dc45b0a4d3eb8
Normal-9798.png,Normal,Normal/images/Normal-9798.png,Normal/masks/Normal-9798.png,True,G19380,train,8a71a160046c656279b53b1ff7dff77b,dd2f461d0a2cbde0
Normal-9799.png,Normal,Normal/images/Normal-9799.png,Normal/masks/Normal-9799.png,True,G19381,train,a181132ce1e3b2a1da16f9ecde9e0209,dc7ec34b19062bb8
Normal-98.png,Normal,Normal/images/Normal-98.png,Normal/masks/Normal-98.png,True,G19382,val,1f7d9e3eff5d146839426d6b9ecfa26f,d07cc71ad92b3c64
Normal-980.png,Normal,Normal/images/Normal-980.png,Normal/masks/Normal-980.png,True,G19383,test,a2c2b24b3b9fafb00e9f5a5b9658a4d0,9105e31eb8693d79
Normal-9800.png,Normal,Normal/images/Normal-9800.png,Normal/masks/Normal-9800.png,True,G19384,train,da80a9bc435e09f23bf199fadd8ecfe5,c90ec74b3c2d2d3c
Normal-9801.png,Normal,Normal/images/Normal-9801.png,Normal/masks/Normal-9801.png,True,G19385,train,76d4284486c67dc4b91b4489bd606570,947c070b0b4f0f3f
Normal-9802.png,Normal,Normal/images/Normal-9802.png,Normal/masks/Normal-9802.png,True,G19386,train,ae0a9d2523fc02a5ef41a1d7ae7aff6b,91795a575a1b5a4a
Normal-9803.png,Normal,Normal/images/Normal-9803.png,Normal/masks/Normal-9803.png,True,G19387,train,b6ae96c61ed7104f70dceee106df16f8,d17b56141a0d7b1d
Normal-9804.png,Normal,Normal/images/Normal-9804.png,Normal/masks/Normal-9804.png,True,G19388,train,0b8d5ecefe16947a0d1d84822da517cd,8979875a194d5f38
Normal-9805.png,Normal,Normal/images/Normal-9805.png,Normal/masks/Normal-9805.png,True,G19389,train,ad9a1096dee894e6a6152b87046b5b37,d52846c71f0aed4b
Normal-9806.png,Normal,Normal/images/Normal-9806.png,Normal/masks/Normal-9806.png,True,G19390,train,7c6d71f8138202edec9ab9c3f32466f0,813896c6181f6fbb
Normal-9807.png,Normal,Normal/images/Normal-9807.png,Normal/masks/Normal-9807.png,True,G19391,train,e107c5d61040dfdca6d5639d6fc02230,d02d52435f4a5f4b
Normal-9808.png,Normal,Normal/images/Normal-9808.png,Normal/masks/Normal-9808.png,True,G19392,train,a07fe4caaa256441ee1b6de2bf1be326,9c49434b4d1e4f3e
Normal-9809.png,Normal,Normal/images/Normal-9809.png,Normal/masks/Normal-9809.png,True,G19393,train,c60b69efac342d6c6adddff2ddd82b26,887d861a195f1a5f
Normal-981.png,Normal,Normal/images/Normal-981.png,Normal/masks/Normal-981.png,True,G19394,train,f76c27b2ee8cee9be2b642c8caacab52,8128c65ac9cb5bd7
Normal-9810.png,Normal,Normal/images/Normal-9810.png,Normal/masks/Normal-9810.png,True,G19395,train,9311c987eab46605a3b014822edec637,d13d4e867b39e086
Normal-9811.png,Normal,Normal/images/Normal-9811.png,Normal/masks/Normal-9811.png,True,G19396,train,5e5b931f428f144cb66d02e3f405e81d,d43c42c3c34b0fde
Normal-9812.png,Normal,Normal/images/Normal-9812.png,Normal/masks/Normal-9812.png,True,G19397,train,c30e2a06aa3f0d2434a1a89278d96269,d93bd61618199b1b
Normal-9813.png,Normal,Normal/images/Normal-9813.png,Normal/masks/Normal-9813.png,True,G19398,val,555c106a10e20618b2922e938bf11a5d,892d465a1a455ff9
Normal-9814.png,Normal,Normal/images/Normal-9814.png,Normal/masks/Normal-9814.png,True,G19399,val,7a782f10605a78e17bb2e4848a33b5fb,813b5646583b793b
Normal-9815.png,Normal,Normal/images/Normal-9815.png,Normal/masks/Normal-9815.png,True,G19400,test,192efcc4e2b05318f5c774b7c8af3c5f,997bd71218475f18
Normal-9816.png,Normal,Normal/images/Normal-9816.png,Normal/masks/Normal-9816.png,True,G19401,test,5395bf99934e007cad7f219612bcc50d,dd2c53c30b5d0e4e
Normal-9817.png,Normal,Normal/images/Normal-9817.png,Normal/masks/Normal-9817.png,True,G19402,train,c506c9897c6a01c11d33357fdb6681fb,817b461a1b5f6a39
Normal-9818.png,Normal,Normal/images/Normal-9818.png,Normal/masks/Normal-9818.png,True,G19403,train,628ffa84fb24f1b13ba4003b9db59222,9969ce06131eee78
Normal-9819.png,Normal,Normal/images/Normal-9819.png,Normal/masks/Normal-9819.png,True,G19404,train,7ee8afcd610a325235b7b2b10b413f96,995906161a5d5f1f
Normal-982.png,Normal,Normal/images/Normal-982.png,Normal/masks/Normal-982.png,True,G19405,train,b86be9454a819a214904024cb3439538,9197c71ad84934eb
Normal-9820.png,Normal,Normal/images/Normal-9820.png,Normal/masks/Normal-9820.png,True,G19406,train,e0f00923405cf8de1bdb8e0aa38c3bb5,dc3d821b554f0fc8
Normal-9821.png,Normal,Normal/images/Normal-9821.png,Normal/masks/Normal-9821.png,True,G19407,train,4a940f0afbb09d58c61aecadd3d42306,dc79d319031e299b
Normal-9822.png,Normal,Normal/images/Normal-9822.png,Normal/masks/Normal-9822.png,True,G19408,train,e8d06f22c84185ed1fb2836d7bbd8511,f92dc412554a1f5e
Normal-9823.png,Normal,Normal/images/Normal-9823.png,Normal/masks/Normal-9823.png,True,G19409,train,1a45eecb715991cc9987875b496683ae,c07b4742161b5f1f
Normal-9824.png,Normal,Normal/images/Normal-9824.png,Normal/masks/Normal-9824.png,True,G19410,train,cf458e9ad26e08bc1e629db3d846918d,9178c6161b5f5e1a
Normal-9825.png,Normal,Normal/images/Normal-9825.png,Normal/masks/Normal-9825.png,True,G19411,val,90070a96a774172ad1e11267fd061990,8179065a79ed6939
Normal-9826.png,Normal,Normal/images/Normal-9826.png,Normal/masks/Normal-9826.png,True,G19412,train,f319b73bc845e020634c87ef79172fe1,896985161edf5e98
Normal-9827.png,Normal,Normal/images/Normal-9827.png,Normal/masks/Normal-9827.png,True,G19413,train,6e444ade57ad2e2d1e8a68447ccfcfbf,d92b471e184d1d3d
Normal-9828.png,Normal,Normal/images/Normal-9828.png,Normal/masks/Normal-9828.png,True,G19414,train,f32e02031ad3058b3dba6cc3fa8dccc9,887945861f1d4f1f
Normal-9829.png,Normal,Normal/images/Normal-9829.png,Normal/masks/Normal-9829.png,True,G19415,train,38bdc4d7aa0fe8bb3ba8ca5918fef11d,99788616155e7e5a
Normal-983.png,Normal,Normal/images/Normal-983.png,Normal/masks/Normal-983.png,True,G19416,val,a9afe698128c7848ec019aa6f384c944,d57f871b8d0c2c86
Normal-9830.png,Normal,Normal/images/Normal-9830.png,Normal/masks/Normal-9830.png,True,G19417,val,444093aa56d4f24852539415dd1230b9,c43b515c0e4f1e2f
Normal-9831.png,Normal,Normal/images/Normal-9831.png,Normal/masks/Normal-9831.png,True,G19418,test,d82de578a8262025222ab3fd4b413121,996807161f0f5e5d
Normal-9832.png,Normal,Normal/images/Normal-9832.png,Normal/masks/Normal-9832.png,True,G19419,test,bc541d701432bc3e54194d4c8ff9ed60,913b5e1869876397
Normal-9833.png,Normal,Normal/images/Normal-9833.png,Normal/masks/Normal-9833.png,True,G19420,train,f6a906781da8ab3bd346f6b1987279c5,d42f075f1a0e3d19
Normal-9834.png,Normal,Normal/images/Normal-9834.png,Normal/masks/Normal-9834.png,True,G19421,train,ec4229304c8f8411242d958a580ee4de,dc4cc30fa86cbce4
Normal-9835.png,Normal,Normal/images/Normal-9835.png,Normal/masks/Normal-9835.png,True,G19422,test,ca03f8f73180f6ea42a9da205437793c,dd2a425f0968bdb4
Normal-9836.png,Normal,Normal/images/Normal-9836.png,Normal/masks/Normal-9836.png,True,G19423,train,5403556ab8187a6f2abd7089a4afe0fc,812a8686da9f573b
Normal-9837.png,Normal,Normal/images/Normal-9837.png,Normal/masks/Normal-9837.png,True,G19424,val,6032b46674105348fafaf218f127a210,dd3cc25b0b4c2d3c
Normal-9838.png,Normal,Normal/images/Normal-9838.png,Normal/masks/Normal-9838.png,True,G19425,train,9923ae177548a7db635af0707a9f61eb,9978961b1d1aba96
Normal-9839.png,Normal,Normal/images/Normal-9839.png,Normal/masks/Normal-9839.png,True,G19426,train,9e4110e695a6b580ff4c07d21cf79dbb,9968565f1a0c7a3d
Normal-984.png,Normal,Normal/images/Normal-984.png,Normal/masks/Normal-984.png,True,G19427,train,59630df03275a28961925e66ff65b345,996d875a9a4f5825
Normal-9840.png,Normal,Normal/images/Normal-9840.png,Normal/masks/Normal-9840.png,True,G19428,train,86dcced9f2c2280b81b8c33658031d5b,cd3dc24d4d0e2cf4
Normal-9841.png,Normal,Normal/images/Normal-9841.png,Normal/masks/Normal-9841.png,True,G19429,train,1123cd16c032858a446960867ca1a763,d0bd2f4a3f34d032
Normal-9842.png,Normal,Normal/images/Normal-9842.png,Normal/masks/Normal-9842.png,True,G19430,train,1460a8bb78948734b5d0361829fcb512,996e1e594d78c3d0
Normal-9843.png,Normal,Normal/images/Normal-9843.png,Normal/masks/Normal-9843.png,True,G19431,val,96dc96170b0bdf0f6b4d2c02772c5e9b,d43c43091b6f3c3d
Normal-9844.png,Normal,Normal/images/Normal-9844.png,Normal/masks/Normal-9844.png,True,G19432,train,0bde1f27619ef560a275d8dbc54601d4,d83947de5a015a1f
Normal-9845.png,Normal,Normal/images/Normal-9845.png,Normal/masks/Normal-9845.png,True,G19433,train,1deb250d91c5dfad9931621ec6616c39,d42ad34f4e486d3a
Normal-9846.png,Normal,Normal/images/Normal-9846.png,Normal/masks/Normal-9846.png,True,G19434,val,d35bced31e8c02b1dba8fd95caf8e49d,de35d95142e9494b
Normal-9847.png,Normal,Normal/images/Normal-9847.png,Normal/masks/Normal-9847.png,True,G19435,test,86ba50176dbb77ba82ec1fce4356cad7,9869071e4a1f5a5f
Normal-9848.png,Normal,Normal/images/Normal-9848.png,Normal/masks/Normal-9848.png,True,G19436,train,18ec1ac777c6c406966a8f29f238114d,dc2cd34b1f420f1e
Normal-9849.png,Normal,Normal/images/Normal-9849.png,Normal/masks/Normal-9849.png,True,G19437,test,da7c0969344930a1d5d9a66cdf91918c,c83d430f19f8687d
Normal-985.png,Normal,Normal/images/Normal-985.png,Normal/masks/Normal-985.png,True,G19438,train,68809b3d2c1ab40d5d18d25d7ddc52dd,907c873bbd4d183c
Normal-9850.png,Normal,Normal/images/Normal-9850.png,Normal/masks/Normal-9850.png,True,G19439,train,3e45a220ed22e3b056c1541616febc97,d979c45a5d4a1a4d
Normal-9851.png,Normal,Normal/images/Normal-9851.png,Normal/masks/Normal-9851.png,True,G19440,train,e3ee3af0fcda02c66095b96fe16ded86,917ac6061d165e5f
Normal-9852.png,Normal,Normal/images/Normal-9852.png,Normal/masks/Normal-9852.png,True,G19441,test,41ed3d4d09737449a16ade155da6855b,9139c6565a594b5b
Normal-9853.png,Normal,Normal/images/Normal-9853.png,Normal/masks/Normal-9853.png,True,G19442,train,931206a0253547ce3e904e0555024769,813d064d7be06df1
Normal-9854.png,Normal,Normal/images/Normal-9854.png,Normal/masks/Normal-9854.png,True,G19443,train,7d9e2afc74d8bbbe88367af3cb293de4,dd3fc04b1b450f1c
Normal-9855.png,Normal,Normal/images/Normal-9855.png,Normal/masks/Normal-9855.png,True,G19444,train,34c80bd265ec639c2721bc5c9449389f,817a171e5879b978
Normal-9856.png,Normal,Normal/images/Normal-9856.png,Normal/masks/Normal-9856.png,True,G19445,train,962b31d10b43b8ca3369d731eab8a22b,997d471b54471a4d
Normal-9857.png,Normal,Normal/images/Normal-9857.png,Normal/masks/Normal-9857.png,True,G19446,train,99528a3430c8a41a9d74436df4627f37,817887ba191d3d5d
Normal-9858.png,Normal,Normal/images/Normal-9858.png,Normal/masks/Normal-9858.png,True,G19447,train,b3f5030d3e991a20cef726d54e4da88a,817847d65b974a99
Normal-9859.png,Normal,Normal/images/Normal-9859.png,Normal/masks/Normal-9859.png,True,G19448,train,47d5aa0a05dc0f7bd021bf6b05db580c,d96cc61b054f1b4d
Normal-986.png,Normal,Normal/images/Normal-986.png,Normal/masks/Normal-986.png,True,G19449,train,1f3afc0f368440ae0de5ef7faea75bb3,c02de23ee86d3c69
Normal-9860.png,Normal,Normal/images/Normal-9860.png,Normal/masks/Normal-9860.png,True,G19450,train,d7a190d39f877ef85d5c543b4ffb524f,dd3d425b8b0a0d9b
Normal-9861.png,Normal,Normal/images/Normal-9861.png,Normal/masks/Normal-9861.png,True,G19451,train,0b3c2b34a35b4e34db3cf703244229db,892b86575d5e1a5a
Normal-9862.png,Normal,Normal/images/Normal-9862.png,Normal/masks/Normal-9862.png,True,G19452,train,31e26491d5a776df41bdfbd1a823a7a9,d80a0a1a7c76f575
Normal-9863.png,Normal,Normal/images/Normal-9863.png,Normal/masks/Normal-9863.png,True,G19453,train,5935d7707ec70e2add794a272429742b,d87dc21b5d48594d
Normal-9864.png,Normal,Normal/images/Normal-9864.png,Normal/masks/Normal-9864.png,True,G19454,train,cb9c7ce89862c4fb0c9f5fff903194ab,887f85161d4f1b1d
Normal-9865.png,Normal,Normal/images/Normal-9865.png,Normal/masks/Normal-9865.png,True,G19455,val,dce66f87b8fa15a87f907ce2d40b759b,dd2cc20b1d843afd
Normal-9866.png,Normal,Normal/images/Normal-9866.png,Normal/masks/Normal-9866.png,True,G19456,val,7711fc2614e293a321146117e614e7ca,dc0ac30b0e4b3f2f
Normal-9867.png,Normal,Normal/images/Normal-9867.png,Normal/masks/Normal-9867.png,True,G19457,val,ac9fa4f9270a148543ce40dfe69c6878,9d6ed05b414f1a0f
Normal-9868.png,Normal,Normal/images/Normal-9868.png,Normal/masks/Normal-9868.png,True,G19458,test,090c3a16834fd1ae262736103a393bec,d82dc61a1f0d4d3d
Normal-9869.png,Normal,Normal/images/Normal-9869.png,Normal/masks/Normal-9869.png,True,G19459,test,a453d660b208c1c0bb9a7f6f8b5d87b2,cc3dc24b5f420f1d
Normal-987.png,Normal,Normal/images/Normal-987.png,Normal/masks/Normal-987.png,True,G19460,train,c39853da977d13888acd4798e9af73e4,997a461a587a79e1
Normal-9870.png,Normal,Normal/images/Normal-9870.png,Normal/masks/Normal-9870.png,True,G19461,test,7679d18c08756975a168e02af9903b14,807f161c1b4f3979
Normal-9871.png,Normal,Normal/images/Normal-9871.png,Normal/masks/Normal-9871.png,True,G19462,train,b1582561836d595f5e3a13be206e0ddb,d13dc243191b4f3e
Normal-9872.png,Normal,Normal/images/Normal-9872.png,Normal/masks/Normal-9872.png,True,G19463,val,ef5298f0c227a69815e4b71999d9b717,9979051e144f1f5e
Normal-9873.png,Normal,Normal/images/Normal-9873.png,Normal/masks/Normal-9873.png,True,G19464,val,949207d5acbe10ef9b60950b5e456812,91e8de56170e1a1f
Normal-9874.png,Normal,Normal/images/Normal-9874.png,Normal/masks/Normal-9874.png,True,G19465,train,ac3f9d9f61d1cd134559176b0b1a7a3e,8178571e1a0f7a3b
Normal-9875.png,Normal,Normal/images/Normal-9875.png,Normal/masks/Normal-9875.png,True,G19466,test,c6ed8d50032e95da012962a55c95a51c,b86dc61ec71e104f
Normal-9876.png,Normal,Normal/images/Normal-9876.png,Normal/masks/Normal-9876.png,True,G19467,train,d083f0bca16b6bac6442b196380c0fe0,837f94101f1b3b1b
Normal-9877.png,Normal,Normal/images/Normal-9877.png,Normal/masks/Normal-9877.png,True,G19468,train,3c175c30606c673e433a91a2a1d26fc1,9169860e115f5e7e
Normal-9878.png,Normal,Normal/images/Normal-9878.png,Normal/masks/Normal-9878.png,True,G19469,train,bb37bb5816fe083af0e304d2a03934ca,983bc7969e481e1e
Normal-9879.png,Normal,Normal/images/Normal-9879.png,Normal/masks/Normal-9879.png,True,G19470,train,37808388c4dbe8848f1295c0c0ea15df,817b05861a1f5d5f
Normal-988.png,Normal,Normal/images/Normal-988.png,Normal/masks/Normal-988.png,True,G19471,train,79e6b0e29bfee6487dcd7add4c7d5fff,896fc74a4f4519c9
Normal-9880.png,Normal,Normal/images/Normal-9880.png,Normal/masks/Normal-9880.png,True,G19472,train,9c7b6e23d263c4d36d7f88a14f39635a,81781e1e5f485e79
Normal-9881.png,Normal,Normal/images/Normal-9881.png,Normal/masks/Normal-9881.png,True,G19473,train,7f7c281af671474e84677f4e04b522b6,d82952575b4e0f39
Normal-9882.png,Normal,Normal/images/Normal-9882.png,Normal/masks/Normal-9882.png,True,G19474,train,e53be0d340750530cd33e5d1d63a46cf,9979061e105f595f
Normal-9883.png,Normal,Normal/images/Normal-9883.png,Normal/masks/Normal-9883.png,True,G19475,train,8ec442e0d9fc66201e4130604ede86f7,c17b47065f0a5f0b
Normal-9884.png,Normal,Normal/images/Normal-9884.png,Normal/masks/Normal-9884.png,True,G19476,train,eb80467aab8231ca6947f1d8d2656f2a,d91d861e1c5b195b
Normal-9885.png,Normal,Normal/images/Normal-9885.png,Normal/masks/Normal-9885.png,True,G19477,train,1bb45a31ea9b8d34daa5680efc0f82f3,d92e461a647af8e1
Normal-9886.png,Normal,Normal/images/Normal-9886.png,Normal/masks/Normal-9886.png,True,G19478,train,bc0d65cb212328aa49ee40ebd10c3f07,957807471a5f0f1d
Normal-9887.png,Normal,Normal/images/Normal-9887.png,Normal/masks/Normal-9887.png,True,G19479,val,4221d738d922ea5a138deb956f6905ee,d129560e5b18ef4b
Normal-9888.png,Normal,Normal/images/Normal-9888.png,Normal/masks/Normal-9888.png,True,G19480,test,08b886ad5cb5e00af1bc62c6c94ba353,d939071fd44e1a6a
Normal-9889.png,Normal,Normal/images/Normal-9889.png,Normal/masks/Normal-9889.png,True,G19481,test,56ba159a2b48fd8b35025eb260e8a10a,dc2cd0530b554b5f
Normal-989.png,Normal,Normal/images/Normal-989.png,Normal/masks/Normal-989.png,True,G19482,train,f78d255ecfd43fa29dae8ddcb4fb6453,91e5c63a98cd39a5
Normal-9890.png,Normal,Normal/images/Normal-9890.png,Normal/masks/Normal-9890.png,True,G19483,train,b86b8686e2f72defef056c755f65f2f3,c93d861b1c4f5b1c
Normal-9891.png,Normal,Normal/images/Normal-9891.png,Normal/masks/Normal-9891.png,True,G19484,train,73964310fcffba32eaaa5c6aa016c2df,dd2e430b4f4e0c2f
Normal-9892.png,Normal,Normal/images/Normal-9892.png,Normal/masks/Normal-9892.png,True,G19485,train,5014995546df3796ef6a134bc8f4f5d0,9979c75a1b054e39
Normal-9893.png,Normal,Normal/images/Normal-9893.png,Normal/masks/Normal-9893.png,True,G19486,train,5533babec5aba3afb7d6336f3f4a1c53,9868071e5ac97f78
Normal-9894.png,Normal,Normal/images/Normal-9894.png,Normal/masks/Normal-9894.png,True,G19487,train,9245ee875401de1f81b03cd24b5ca08b,d13f461b1b4c69b8
Normal-9895.png,Normal,Normal/images/Normal-9895.png,Normal/masks/Normal-9895.png,True,G19488,train,ef26dbefe26f2744c42535dc18e97a7c,992d061e1b4d7d59
Normal-9896.png,Normal,Normal/images/Normal-9896.png,Normal/masks/Normal-9896.png,True,G19489,train,72c5ca61a4b7cbe601744800a13e95b1,dd7d50d3131d4d09
Normal-9897.png,Normal,Normal/images/Normal-9897.png,Normal/masks/Normal-9897.png,True,G19490,train,95357dbe695f623d809a5b71b4fa6c21,d978c31bd41f504d
Normal-9898.png,Normal,Normal/images/Normal-9898.png,Normal/masks/Normal-9898.png,True,G19491,train,d459278caf6254fb064e487f4b2acfa1,c02f8319144f7f76
Normal-9899.png,Normal,Normal/images/Normal-9899.png,Normal/masks/Normal-9899.png,True,G19492,train,73060f5d8e58112ff79661086f0603ac,d97cc61b034c39ad
Normal-99.png,Normal,Normal/images/Normal-99.png,Normal/masks/Normal-99.png,True,G19493,train,71a7a5d3467aa21868e6a906478ec8ee,943bc31bbcf03c92
Normal-990.png,Normal,Normal/images/Normal-990.png,Normal/masks/Normal-990.png,True,G19494,val,0c65dad8e1838c43d5a29ebdafaddeca,9009c71ece491def
Normal-9900.png,Normal,Normal/images/Normal-9900.png,Normal/masks/Normal-9900.png,True,G19495,val,44ea1a658d02fc4fd15d6df72085a990,cc34c34b4a4b2f3d
Normal-9901.png,Normal,Normal/images/Normal-9901.png,Normal/masks/Normal-9901.png,True,G19496,train,7b6ed0755d4058aaa524bbe2e22069c4,99a4471a471f471f
Normal-9902.png,Normal,Normal/images/Normal-9902.png,Normal/masks/Normal-9902.png,True,G19497,val,3d3d2e6d041f49969921b298a9ca1f93,9c28530b434f1f5f
Normal-9903.png,Normal,Normal/images/Normal-9903.png,Normal/masks/Normal-9903.png,True,G19498,test,09b35998dc70e67168ff53b25a041eb2,d939d20e482fbde0
Normal-9904.png,Normal,Normal/images/Normal-9904.png,Normal/masks/Normal-9904.png,True,G19499,train,6475315bdb7836eeb97ada523b84ae07,9939c61a594d3ae5
Normal-9905.png,Normal,Normal/images/Normal-9905.png,Normal/masks/Normal-9905.png,True,G19500,test,43b6088af4e18ff87207ee02f7ba7081,993b571e1e495a1a
Normal-9906.png,Normal,Normal/images/Normal-9906.png,Normal/masks/Normal-9906.png,True,G19501,test,10d24520f0e0ee55faa03f665042d000,dc2cd2531bd44a1f
Normal-9907.png,Normal,Normal/images/Normal-9907.png,Normal/masks/Normal-9907.png,True,G19502,train,2ecfc274f11b8fb995fef230a1b784af,e8f9974e901f4e90
Normal-9908.png,Normal,Normal/images/Normal-9908.png,Normal/masks/Normal-9908.png,True,G19503,train,8fc6e117e1236d79fc0330df28a8bc9d,dd30d30fc70b0b4b
Normal-9909.png,Normal,Normal/images/Normal-9909.png,Normal/masks/Normal-9909.png,True,G19504,train,8ef5a95457b06632ac11590ec521d8fa,c92dc61e951e9c4e
Normal-991.png,Normal,Normal/images/Normal-991.png,Normal/masks/Normal-991.png,True,G19505,train,6d2f7253e997fdcbaa82a4af1df13753,913dc61b9ce13996
Normal-9910.png,Normal,Normal/images/Normal-9910.png,Normal/masks/Normal-9910.png,True,G19506,train,1dd1cb34f70f850bd1b97c3af3d1174e,dc3dd2434d4a0f1d
Normal-9911.png,Normal,Normal/images/Normal-9911.png,Normal/masks/Normal-9911.png,True,G19507,train,9939fad481d965eebb34a11349acd7f1,917bc6161a151f5b
Normal-9912.png,Normal,Normal/images/Normal-9912.png,Normal/masks/Normal-9912.png,True,G19508,train,de01c532d4e2eb16477e5da3baebccbc,897d0616174f195f
Normal-9913.png,Normal,Normal/images/Normal-9913.png,Normal/masks/Normal-9913.png,True,G19509,train,1dfd1d38ebf223910b2e7d0d957451d4,fd28571fc51b5904
Normal-9914.png,Normal,Normal/images/Normal-9914.png,Normal/masks/Normal-9914.png,True,G19510,train,52f0dc6df68e6885708266699a331d21,8179d2865e1a5e1f
Normal-9915.png,Normal,Normal/images/Normal-9915.png,Normal/masks/Normal-9915.png,True,G19511,train,e2d7d57d787861637e41b29b9f72b063,9568074b594f7e68
Normal-9916.png,Normal,Normal/images/Normal-9916.png,Normal/masks/Normal-9916.png,True,G19512,train,1df4e759d0a2769f7dc024c963adf180,d92bc6061fc49b99
Normal-9917.png,Normal,Normal/images/Normal-9917.png,Normal/masks/Normal-9917.png,True,G19513,train,334da62ce2102f1359eb86c55616b81e,d93997561e1c4a1b
Normal-9918.png,Normal,Normal/images/Normal-9918.png,Normal/masks/Normal-9918.png,True,G19514,train,f6a87c4cc0796db7fd015649b9eacefb,d03c0f4a19eee4b5
Normal-9919.png,Normal,Normal/images/Normal-9919.png,Normal/masks/Normal-9919.png,True,G19515,train,17376c93298547c97b74731cc9c2212b,887c465a194dfeac
Normal-992.png,Normal,Normal/images/Normal-992.png,Normal/masks/Normal-992.png,True,G19516,val,55ad7a5c7c29579ad5874b65d7b8b079,914dc65ad9c919c7
Normal-9920.png,Normal,Normal/images/Normal-9920.png,Normal/masks/Normal-9920.png,True,G19517,val,88b9977fb0d6093c3a1048c29578cbe4,c03d42575a1e4f5b
Normal-9921.png,Normal,Normal/images/Normal-9921.png,Normal/masks/Normal-9921.png,True,G19518,val,4aa556c6d902d46d1a6cad41919bd3fb,d03c8f43b95eec30
Normal-9922.png,Normal,Normal/images/Normal-9922.png,Normal/masks/Normal-9922.png,True,G19519,test,f46b0accff96465e6e580930203df82b,dd2cd0431bd7421f
Normal-9923.png,Normal,Normal/images/Normal-9923.png,Normal/masks/Normal-9923.png,True,G19520,test,2f2891f77d48f17e7c4e8e439b02d228,d97b8606194d5e3a
Normal-9924.png,Normal,Normal/images/Normal-9924.png,Normal/masks/Normal-9924.png,True,G19521,test,841f08dee3c82f0e883b3b876d55470b,d569421b484e1f5f
Normal-9925.png,Normal,Normal/images/Normal-9925.png,Normal/masks/Normal-9925.png,True,G19522,train,552426b351bbaadc03efb58c16e2be5f,91695e56185a9d79
Normal-9926.png,Normal,Normal/images/Normal-9926.png,Normal/masks/Normal-9926.png,True,G19523,train,291a5373b0cb36647b6002f1fa4ed09a,cd2cc24b0d5d2eec
Normal-9927.png,Normal,Normal/images/Normal-9927.png,Normal/masks/Normal-9927.png,True,G19524,train,8b0258f9ad72d8a305976f1daf7dbe69,953c06821d5ecf9e
Normal-9928.png,Normal,Normal/images/Normal-9928.png,Normal/masks/Normal-9928.png,True,G19525,train,0e77deeaaa4e67966be891f55bf9640c,d97fc4421f844f19
Normal-9929.png,Normal,Normal/images/Normal-9929.png,Normal/masks/Normal-9929.png,True,G19526,train,ed9e992da1bdf4f2b9eea071bbec8c7b,9929571a1e4f1b1d
Normal-993.png,Normal,Normal/images/Normal-993.png,Normal/masks/Normal-993.png,True,G19527,train,7e18d541f79eba92f15a7dde0fa3fd87,913fc61ab8e03997
Normal-9930.png,Normal,Normal/images/Normal-9930.png,Normal/masks/Normal-9930.png,True,G19528,train,6ebe3feb7d43b126155939ea9bd17606,8979861e594d1e79
Normal-9931.png,Normal,Normal/images/Normal-9931.png,Normal/masks/Normal-9931.png,True,G19529,train,ef3c3584d0de8499702be4909dbff122,dd2cd20f4d4a1be4
Normal-9932.png,Normal,Normal/images/Normal-9932.png,Normal/masks/Normal-9932.png,True,G19530,train,0d8d4883818b9f5882eeb79e65a4367c,dc7dc25b5d41184d
Normal-9933.png,Normal,Normal/images/Normal-9933.png,Normal/masks/Normal-9933.png,True,G19531,train,ebd6bf7803b7aaf21ce0314f8131a5e8,99065a7976736525
Normal-9934.png,Normal,Normal/images/Normal-9934.png,Normal/masks/Normal-9934.png,True,G19532,train,15f902e5b721dd913104b576fbb9d400,c13ad65b5c4a5b19
Normal-9935.png,Normal,Normal/images/Normal-9935.png,Normal/masks/Normal-9935.png,True,G19533,val,1410c5a80770b4e176b0c252d353d0f5,884dc61b5d4d5a79
Normal-9936.png,Normal,Normal/images/Normal-9936.png,Normal/masks/Normal-9936.png,True,G19534,train,1688594cb8cc7a351b3e9ce95f0a42f5,dc2c83821f45de5e
Normal-9937.png,Normal,Normal/images/Normal-9937.png,Normal/masks/Normal-9937.png,True,G19535,test,3f55d435da3132096540e00f49bfde8f,bd7cc61259c71b09
Normal-9938.png,Normal,Normal/images/Normal-9938.png,Normal/masks/Normal-9938.png,True,G19536,val,4ff69b7f3e41367ad41016f4e6b7f689,81f8c65b14c67bd4
Normal-9939.png,Normal,Normal/images/Normal-9939.png,Normal/masks/Normal-9939.png,True,G19537,train,51253ae22ada13d9d87496ff56ad15a4,8078c7165a575f5a
Normal-994.png,Normal,Normal/images/Normal-994.png,Normal/masks/Normal-994.png,True,G19538,train,c996245e858ef86826e426f35cec8b54,c083efcb68e0cbb8
Normal-9940.png,Normal,Normal/images/Normal-9940.png,Normal/masks/Normal-9940.png,True,G19539,train,b051c4ee86ba2c72fbf96c5c11c6ac61,886e570b1f6d2a35
Normal-9941.png,Normal,Normal/images/Normal-9941.png,Normal/masks/Normal-9941.png,True,G19540,train,40a04027ca1100d04d4c762be931e45f,897f861a1e4e3ab8
Normal-9942.png,Normal,Normal/images/Normal-9942.png,Normal/masks/Normal-9942.png,True,G19541,test,cd97d288b7b93c4c89fa29768c11488b,d83b85561e4f5e18
Normal-9943.png,Normal,Normal/images/Normal-9943.png,Normal/masks/Normal-9943.png,True,G19542,test,066422a3a4c5d7f97f73161aa50d7379,dd2c460a196ce4fb
Normal-9944.png,Normal,Normal/images/Normal-9944.png,Normal/masks/Normal-9944.png,True,G19543,val,b977896e3a8a5f59ff5a3ff0d318b2bd,89fa871a93071e3b
Normal-9945.png,Normal,Normal/images/Normal-9945.png,Normal/masks/Normal-9945.png,True,G19544,test,f5ffc3a61a2931f5c7af858d5ff246bb,993b46131f0f2b39
Normal-9946.png,Normal,Normal/images/Normal-9946.png,Normal/masks/Normal-9946.png,True,G19545,test,1522a8207fddcdf9cd9c744fd9e9aef1,dd2f42d32b402b9d
Normal-9947.png,Normal,Normal/images/Normal-9947.png,Normal/masks/Normal-9947.png,True,G19546,train,a8f5015827cc5c1f20a417167763f6eb,817b065a5f095f1d
Normal-9948.png,Normal,Normal/images/Normal-9948.png,Normal/masks/Normal-9948.png,True,G19547,train,dc3be3fc6cf7f1bb6f1d8d0fb0b678a4,d9bdd217c2495741
Normal-9949.png,Normal,Normal/images/Normal-9949.png,Normal/masks/Normal-9949.png,True,G19548,train,d0fbf467d3f776bd6b697375604f5ea0,d17c870a1a4f783d
Normal-995.png,Normal,Normal/images/Normal-995.png,Normal/masks/Normal-995.png,True,G19549,train,ed95d33df2f11bbc434fe11c69046222,944dc31bcb49dce2
Normal-9950.png,Normal,Normal/images/Normal-9950.png,Normal/masks/Normal-9950.png,True,G19550,train,3a4900558f3f197e44d40a0bf0b91e56,d53c06821b1eeeba
Normal-9951.png,Normal,Normal/images/Normal-9951.png,Normal/masks/Normal-9951.png,True,G19551,train,573310c4f42f4bb99a1567a3b81f896c,9968065a5b6e79e4
Normal-9952.png,Normal,Normal/images/Normal-9952.png,Normal/masks/Normal-9952.png,True,G19552,train,0b26ae8a787b5a7358ebb90d801016c8,dc2c17c21e1e8ed6
Normal-9953.png,Normal,Normal/images/Normal-9953.png,Normal/masks/Normal-9953.png,True,G19553,train,a69c823a4c9a329820b4263fc8bab904,dc2dd24b5f400f1d
Normal-9954.png,Normal,Normal/images/Normal-9954.png,Normal/masks/Normal-9954.png,True,G19554,train,217b9af74d621be73f24baaf1bcef889,9528061e1d5f5c5f
Normal-9955.png,Normal,Normal/images/Normal-9955.png,Normal/masks/Normal-9955.png,True,G19555,train,cdf75f4b478cf7be20f8b82ba7c28cf5,dc2dc20b8fc30b9d
Normal-9956.png,Normal,Normal/images/Normal-9956.png,Normal/masks/Normal-9956.png,True,G19556,train,045bce6c646f1eebbc7ae4cc669e85d8,916f87161f0f1a59
Normal-9957.png,Normal,Normal/images/Normal-9957.png,Normal/masks/Normal-9957.png,True,G19557,train,6e73ae41742e2f1ffaf45eebf70f5389,cc3d434213574f1f
Normal-9958.png,Normal,Normal/images/Normal-9958.png,Normal/masks/Normal-9958.png,True,G19558,val,a9280d03928d0b74fe2f21beafd2bee5,8979c79e5a494d19
Normal-9959.png,Normal,Normal/images/Normal-9959.png,Normal/masks/Normal-9959.png,True,G19559,val,955e5decd4006d7a3685fd1bbd38520d,d17b461a18487f3d
Normal-996.png,Normal,Normal/images/Normal-996.png,Normal/masks/Normal-996.png,True,G19560,val,fad6cdfb2880f312f8875dc1b06d5635,d03ce71f8c4d1ccc
Normal-9960.png,Normal,Normal/images/Normal-9960.png,Normal/masks/Normal-9960.png,True,G19561,test,282520b4399fa9503da50c4d13910000,d12d86921bc55f59
Normal-9961.png,Normal,Normal/images/Normal-9961.png,Normal/masks/Normal-9961.png,True,G19562,test,da9e64b121e497c8ad750dfc9b8cb4a9,dd3df44217680a1f
Normal-9962.png,Normal,Normal/images/Normal-9962.png,Normal/masks/Normal-9962.png,True,G19563,train,99bf00e1a3402fe865a6ed850af6b33c,9909961a5d5d78e5
Normal-9963.png,Normal,Normal/images/Normal-9963.png,Normal/masks/Normal-9963.png,True,G19564,train,ea78c63f231c33a2be68ed80a660e708,dd2ad05b1e554a1e
Normal-9964.png,Normal,Normal/images/Normal-9964.png,Normal/masks/Normal-9964.png,True,G19565,train,9e8c3f8e2237b32a04e3b569f3e5dcd1,91ebc71ac13c394d
Normal-9965.png,Normal,Normal/images/Normal-9965.png,Normal/masks/Normal-9965.png,True,G19566,train,f494951521d39b48697b718b2def3ee4,dd2c02c30e5faaec
Normal-9966.png,Normal,Normal/images/Normal-9966.png,Normal/masks/Normal-9966.png,True,G19567,train,31f1b830fce521750b01a9390234cb4b,dd2a420f4c2d3db5
Normal-9967.png,Normal,Normal/images/Normal-9967.png,Normal/masks/Normal-9967.png,True,G19568,train,a658b9881d0bd95679f2a94962dea2e6,d96f061a455b1d4b
Normal-9968.png,Normal,Normal/images/Normal-9968.png,Normal/masks/Normal-9968.png,True,G19569,train,085bd901a1e2a69c385b230ca7e6673b,c03ccb1c2c3e2d3f
Normal-9969.png,Normal,Normal/images/Normal-9969.png,Normal/masks/Normal-9969.png,True,G19570,train,0be468fba58819ecea8969aba40a13b5,9569ae1e7909eca8
Normal-997.png,Normal,Normal/images/Normal-997.png,Normal/masks/Normal-997.png,True,G19571,train,d124173d4fcdf037c9851efc727207a9,d92dc63a9c693c4c
Normal-9970.png,Normal,Normal/images/Normal-9970.png,Normal/masks/Normal-9970.png,True,G19572,train,28bf900b32eff54925b13b4179a0dc46,dd3f024a0b3c6a3d
Normal-9971.png,Normal,Normal/images/Normal-9971.png,Normal/masks/Normal-9971.png,True,G19573,train,7e14f94eb9cb25aef6a289f6f57c764f,a96c861a5e7a58f8
Normal-9972.png,Normal,Normal/images/Normal-9972.png,Normal/masks/Normal-9972.png,True,G19574,val,5e61a41bde178725d45c2896b347036a,9c4d435b0d4f1e4d
Normal-9973.png,Normal,Normal/images/Normal-9973.png,Normal/masks/Normal-9973.png,True,G19575,val,e72aa12b48a39d034a8517f7a8e648a6,99394f1e1a5bec28
Normal-9974.png,Normal,Normal/images/Normal-9974.png,Normal/masks/Normal-9974.png,True,G19576,test,ac18475f48f4ae6c30d7e6a3a7ce0f0a,997bd61a4d4a390b
Normal-9975.png,Normal,Normal/images/Normal-9975.png,Normal/masks/Normal-9975.png,True,G19577,train,4081ad6d84c3099443c034037288108c,dd2cd0d39b53071a
Normal-9976.png,Normal,Normal/images/Normal-9976.png,Normal/masks/Normal-9976.png,True,G19578,train,8fd9fd0a7ae8b13f180d384bf848c83b,d52bc61e58087f39
Normal-9977.png,Normal,Normal/images/Normal-9977.png,Normal/masks/Normal-9977.png,True,G19579,train,2380291d35f5592dc7627ad676a5cc80,dc39d24b96435f50
Normal-9978.png,Normal,Normal/images/Normal-9978.png,Normal/masks/Normal-9978.png,True,G19580,train,99f869e89e214508198e89f301a9d29b,9979851e05171fdc
Normal-9979.png,Normal,Normal/images/Normal-9979.png,Normal/masks/Normal-9979.png,True,G19581,train,99fafb59c195d496f932715287c93272,dd2ed0535b444d1b
Normal-998.png,Normal,Normal/images/Normal-998.png,Normal/masks/Normal-998.png,True,G19582,train,b0610aefa1fbf40cbe700e8dd55bf3dc,917ce638396d383c
Normal-9980.png,Normal,Normal/images/Normal-9980.png,Normal/masks/Normal-9980.png,True,G19583,train,e5417f85b666a270d1938be870cf6403,dc3d42d30f0e930f
Normal-9981.png,Normal,Normal/images/Normal-9981.png,Normal/masks/Normal-9981.png,True,G19584,test,e921894b02f72d3a644ba5b051e0c700,913d461e585f1b78
Normal-9982.png,Normal,Normal/images/Normal-9982.png,Normal/masks/Normal-9982.png,True,G19585,train,f040399a47d105ca103b2e8a4204cefe,c13906181eed3bbd
Normal-9983.png,Normal,Normal/images/Normal-9983.png,Normal/masks/Normal-9983.png,True,G19586,train,cfbe87b0fc6ee4bc846689acec1f6af2,997dc41a5a0f1a7a
Normal-9984.png,Normal,Normal/images/Normal-9984.png,Normal/masks/Normal-9984.png,True,G19587,val,f7e7b537a26d29915e33db529f4e50a9,fd2dd213c55a5542
Normal-9985.png,Normal,Normal/images/Normal-9985.png,Normal/masks/Normal-9985.png,True,G19588,test,186d3034a4deebff8c32657a13dbb083,9d6f805b09455bcb
Normal-9986.png,Normal,Normal/images/Normal-9986.png,Normal/masks/Normal-9986.png,True,G19589,train,014d689d8d3010d8a9ad1b6c8c933d74,d92955161b5d4f1a
Normal-9987.png,Normal,Normal/images/Normal-9987.png,Normal/masks/Normal-9987.png,True,G19590,train,bbb8e1917036235dfdfeca3a1e529b74,9169ce86185f1e7a
Normal-9988.png,Normal,Normal/images/Normal-9988.png,Normal/masks/Normal-9988.png,True,G19591,train,9302c3df575816d5a2a252bd28886fa6,997dd61a0d5e190e
Normal-9989.png,Normal,Normal/images/Normal-9989.png,Normal/masks/Normal-9989.png,True,G19592,train,8e081ca89c40f91fe97720f7a24a85fc,884f931f4d1e471e
Normal-999.png,Normal,Normal/images/Normal-999.png,Normal/masks/Normal-999.png,True,G19593,train,e6656225fc982a561ac1d6836186c0ef,c02dc71e6dc90fc6
Normal-9990.png,Normal,Normal/images/Normal-9990.png,Normal/masks/Normal-9990.png,True,G19594,train,2b24b7ef1770137fde7a0558ff0b571a,d97dc616c7461a43
Normal-9991.png,Normal,Normal/images/Normal-9991.png,Normal/masks/Normal-9991.png,True,G19595,train,27135aa605fef781b438ec4db83473d9,d13d1a4b3de426c6
Normal-9992.png,Normal,Normal/images/Normal-9992.png,Normal/masks/Normal-9992.png,True,G19596,val,2b4367bfecae12115a2dc7b19b463a37,8979971e188d5f38
Normal-9993.png,Normal,Normal/images/Normal-9993.png,Normal/masks/Normal-9993.png,True,G19597,val,3b10de02cfbe1c55eadaf15e6ce70930,817d07561f0f5f18
Normal-9994.png,Normal,Normal/images/Normal-9994.png,Normal/masks/Normal-9994.png,True,G19598,test,f5fb9687c1c9c0bf340387593f321b29,dd3c825bd147580f
Normal-9995.png,Normal,Normal/images/Normal-9995.png,Normal/masks/Normal-9995.png,True,G19599,train,32b01c219cb87a0674c39fadc3273578,d93d445b1b4d4a3c
Normal-9996.png,Normal,Normal/images/Normal-9996.png,Normal/masks/Normal-9996.png,True,G19600,test,32cc8260546af740b929dd9b1002ced4,8961161a175e3bdb
Normal-9997.png,Normal,Normal/images/Normal-9997.png,Normal/masks/Normal-9997.png,True,G19601,val,f0e959af6b5ae37b02f8bc3d2db661a0,cd0ec24f0d6a3cec
Normal-9998.png,Normal,Normal/images/Normal-9998.png,Normal/masks/Normal-9998.png,True,G19602,test,d3d12bc3b86f60df5f56526496dbc1fd,d979451e1e0d2e2d
Normal-9999.png,Normal,Normal/images/Normal-9999.png,Normal/masks/Normal-9999.png,True,G19603,train,67105a8c76ff1fb45d6a13cc1b5f7fff,d43e53431d0eaf0b
Viral Pneumonia-1.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1.png,Viral Pneumonia/masks/Viral Pneumonia-1.png,True,G19604,train,3158b8fe7878ae73f8718b55ded7fff2,d029e30bfa687e49
Viral Pneumonia-10.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-10.png,Viral Pneumonia/masks/Viral Pneumonia-10.png,True,G19605,train,87593be64c4d400e28ff9b33853631a3,947a4b0f134f5b43
Viral Pneumonia-100.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-100.png,Viral Pneumonia/masks/Viral Pneumonia-100.png,True,G19606,train,9f17f132aff79d1361900626e4db9272,d042c25bc73fc14f
Viral Pneumonia-1000.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1000.png,Viral Pneumonia/masks/Viral Pneumonia-1000.png,True,G19607,train,66205992e18c051d728f76383cad4da0,902dc63ac479f46d
Viral Pneumonia-1001.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1001.png,Viral Pneumonia/masks/Viral Pneumonia-1001.png,True,G19608,train,df72fce3ca7b130ab6ea6e531d8902b3,94a0e31fe03ff04b
Viral Pneumonia-1002.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1002.png,Viral Pneumonia/masks/Viral Pneumonia-1002.png,True,G19609,test,7350fef3ccd26c15ad76f54f8ea6309b,94feaf3a0d3c044e
Viral Pneumonia-1003.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1003.png,Viral Pneumonia/masks/Viral Pneumonia-1003.png,True,G19610,train,b76684ca49727b3faccc74e508aac7de,c02ae24b9e693d6d
Viral Pneumonia-1004.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1004.png,Viral Pneumonia/masks/Viral Pneumonia-1004.png,True,G19611,val,11bc273ec28153e0cc08568ed2f8349e,8131e7963dd2463d
Viral Pneumonia-1005.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1005.png,Viral Pneumonia/masks/Viral Pneumonia-1005.png,True,G19612,train,845ba4ac50c2a586345fb2bb3ab97565,d42ea33b97c31991
Viral Pneumonia-1006.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1006.png,Viral Pneumonia/masks/Viral Pneumonia-1006.png,True,G19613,train,d507991581713650c16e11eb9c93091d,912f7e1c100b3d6f
Viral Pneumonia-1007.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1007.png,Viral Pneumonia/masks/Viral Pneumonia-1007.png,True,G19614,train,bf818f598348b31db221b988a3b414f3,942bd71f091f5c49
Viral Pneumonia-1008.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1008.png,Viral Pneumonia/masks/Viral Pneumonia-1008.png,True,G19615,train,7acf6f6c1d0fce0106d9149fe8802269,946f4b1ecd0d1cc3
Viral Pneumonia-1009.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1009.png,Viral Pneumonia/masks/Viral Pneumonia-1009.png,True,G19616,train,6358aac29071efb2993d2290fd839b82,956dea1cc539344b
Viral Pneumonia-101.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-101.png,Viral Pneumonia/masks/Viral Pneumonia-101.png,True,G19617,train,8c92c2caa58430cc06887582ed973d27,d02de61ae96b3c68
Viral Pneumonia-1010.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1010.png,Viral Pneumonia/masks/Viral Pneumonia-1010.png,True,G19618,train,9e3dafb5dc7e16f49ff50b792b1b3973,95fa10555e1a5f1a
Viral Pneumonia-1011.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1011.png,Viral Pneumonia/masks/Viral Pneumonia-1011.png,True,G19619,val,d36f1e3567a27c75cd2138e6c6ec5229,90aae31bd6cb3ec0
Viral Pneumonia-1012.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1012.png,Viral Pneumonia/masks/Viral Pneumonia-1012.png,True,G19620,val,8c43e8218890e1c9b8e464c21803686a,c06ee30f2d862e6b
Viral Pneumonia-1013.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1013.png,Viral Pneumonia/masks/Viral Pneumonia-1013.png,True,G19621,test,3f07825623faa13398e0f7043b9ed970,d07ec61b9b833991
Viral Pneumonia-1014.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1014.png,Viral Pneumonia/masks/Viral Pneumonia-1014.png,True,G19622,test,a95c40201039ee8561cbfd41ca36e20c,d05b572a1eac343d
Viral Pneumonia-1015.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1015.png,Viral Pneumonia/masks/Viral Pneumonia-1015.png,True,G19623,train,8f576a8ed2a4de17b5f7b431b62f444f,902be75e5e4c282f
Viral Pneumonia-1016.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1016.png,Viral Pneumonia/masks/Viral Pneumonia-1016.png,True,G19624,train,81ff4145c49b9e4fa0d88d1d018dd3ca,d024c31fe74b964b
Viral Pneumonia-1017.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1017.png,Viral Pneumonia/masks/Viral Pneumonia-1017.png,True,G19625,train,9595f522b5c0411415fcefbb6e4cbaec,9454eb3fa6a63aa0
Viral Pneumonia-1018.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1018.png,Viral Pneumonia/masks/Viral Pneumonia-1018.png,True,G19626,train,c034dd88fe1db3c8e22e27ab1ed4105a,c43e871b1b971b19
Viral Pneumonia-1019.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1019.png,Viral Pneumonia/masks/Viral Pneumonia-1019.png,True,G19627,train,01cab5f49c1f9044c301519fee5cb1fb,d07ea77b8739800e
Viral Pneumonia-102.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-102.png,Viral Pneumonia/masks/Viral Pneumonia-102.png,True,G19628,train,4f905c39a62ad9834244eaf884237da0,916de63ea929b948
Viral Pneumonia-1020.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1020.png,Viral Pneumonia/masks/Viral Pneumonia-1020.png,True,G19629,train,9d4274069c180e17aba2d5d551c192c9,d068c69edac65ab1
Viral Pneumonia-1021.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1021.png,Viral Pneumonia/masks/Viral Pneumonia-1021.png,True,G19630,train,cb9e45258f993b0519fe88c4175d140d,9066865a873fe996
Viral Pneumonia-1022.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1022.png,Viral Pneumonia/masks/Viral Pneumonia-1022.png,True,G19631,train,0fa156941a022f51541880451a0f425d,d4a4c31bc14f3cc7
Viral Pneumonia-1023.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1023.png,Viral Pneumonia/masks/Viral Pneumonia-1023.png,True,G19632,train,457a55a1a37f99812fc2ec8dd30785f8,9129e61af9793961
Viral Pneumonia-1024.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1024.png,Viral Pneumonia/masks/Viral Pneumonia-1024.png,True,G19633,train,81e2a42635a8d689547fdc97244d5357,d1a5e21ed24b9ec2
Viral Pneumonia-1025.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1025.png,Viral Pneumonia/masks/Viral Pneumonia-1025.png,True,G19634,train,31fbc123a2e94ffd8b8acd4ded512a2c,9527d71bc728f00e
Viral Pneumonia-1026.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1026.png,Viral Pneumonia/masks/Viral Pneumonia-1026.png,True,G19635,train,f0f9e1d0e4b850b7f1a9b2391c15906f,d0afc21be8693cc3
Viral Pneumonia-1027.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1027.png,Viral Pneumonia/masks/Viral Pneumonia-1027.png,True,G19636,train,1e55344480985e5df29f1e8ff7319e13,d56ce23ba30b9a83
Viral Pneumonia-1028.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1028.png,Viral Pneumonia/masks/Viral Pneumonia-1028.png,True,G19637,train,0ffb052febe50ae01f2da4a67e0d28c2,d0b9e61b524f4d92
Viral Pneumonia-1029.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1029.png,Viral Pneumonia/masks/Viral Pneumonia-1029.png,True,G19638,train,099fff40049880ac4f3da979ccfedf50,9067c61ac13df24f
Viral Pneumonia-103.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-103.png,Viral Pneumonia/masks/Viral Pneumonia-103.png,True,G19639,val,ae11371b771d323d072cc68df0f3433d,91afeabae81e3882
Viral Pneumonia-1030.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1030.png,Viral Pneumonia/masks/Viral Pneumonia-1030.png,True,G19640,val,a7266457d18aaf315b14081e013e51c2,912fc618e969a94f
Viral Pneumonia-1031.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1031.png,Viral Pneumonia/masks/Viral Pneumonia-1031.png,True,G19641,val,3bfc9b2373f6286cc63f26225647b4c1,d014c24fd33db9c3
Viral Pneumonia-1032.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1032.png,Viral Pneumonia/masks/Viral Pneumonia-1032.png,True,G19642,test,ff4ea10b61db3cc3c0956498bf74f7be,d125e20fbeca3cc8
Viral Pneumonia-1033.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1033.png,Viral Pneumonia/masks/Viral Pneumonia-1033.png,True,G19643,test,0b46b0143eb0a668f4eda01b2ecd804c,d024c23fc379bc4b
Viral Pneumonia-1034.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1034.png,Viral Pneumonia/masks/Viral Pneumonia-1034.png,True,G19644,test,d5395817602d4ca74d1c0803bc2ce94b,9135c31fc6ca3cc6
Viral Pneumonia-1035.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1035.png,Viral Pneumonia/masks/Viral Pneumonia-1035.png,True,G19645,train,d13ae516c9452ef73cfe1340ee77bdda,d46fa33b831b8a49
Viral Pneumonia-1036.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1036.png,Viral Pneumonia/masks/Viral Pneumonia-1036.png,True,G19646,train,63970c776201359b0f14038c00ce8bb1,9135c61ed64bbcc2
Viral Pneumonia-1037.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1037.png,Viral Pneumonia/masks/Viral Pneumonia-1037.png,True,G19647,train,34c3a0ac968406d475b67a612f9423c1,d0f8e31be20e6d86
Viral Pneumonia-1038.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1038.png,Viral Pneumonia/masks/Viral Pneumonia-1038.png,True,G19648,train,818d0c6076db4f09d510aa3d2c9d7f39,9138e37af44e6c16
Viral Pneumonia-1039.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1039.png,Viral Pneumonia/masks/Viral Pneumonia-1039.png,True,G19649,test,35a3f744aa938e1c084bc138c41e1f2f,91b5e61bb8ed38a0
Viral Pneumonia-104.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-104.png,Viral Pneumonia/masks/Viral Pneumonia-104.png,True,G19650,train,8e6d98b9f726e86122871d9ffade2fe4,d0e6975b85398738
Viral Pneumonia-1040.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1040.png,Viral Pneumonia/masks/Viral Pneumonia-1040.png,True,G19651,train,f9f2b2880133a60de13a198d6f47bc8e,d47ea73b89972881
Viral Pneumonia-1041.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1041.png,Viral Pneumonia/masks/Viral Pneumonia-1041.png,True,G19652,train,8dedb5255d205020c3b9a03ea2d47171,c41cf24b1fe46c3c
Viral Pneumonia-1042.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1042.png,Viral Pneumonia/masks/Viral Pneumonia-1042.png,True,G19653,train,052660d135baaa32983160d42555a856,9139e31abaca3d38
Viral Pneumonia-1043.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1043.png,Viral Pneumonia/masks/Viral Pneumonia-1043.png,True,G19654,train,b6ea18439de58e5458a56e39240f4947,d47fa71e8a0b1c45
Viral Pneumonia-1044.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1044.png,Viral Pneumonia/masks/Viral Pneumonia-1044.png,True,G19655,train,ab5aaa6732e29fbf5424642bb2806ea0,c06a865aed2d6d87
Viral Pneumonia-1045.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1045.png,Viral Pneumonia/masks/Viral Pneumonia-1045.png,True,G19656,test,117c05dc3459a294847f2a8ca813edc3,d074c71b834bb9c6
Viral Pneumonia-1046.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1046.png,Viral Pneumonia/masks/Viral Pneumonia-1046.png,True,G19657,train,23c6dbfdc9859fbd06fc0d7343f3af7d,d43cc016f70bbd83
Viral Pneumonia-1047.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1047.png,Viral Pneumonia/masks/Viral Pneumonia-1047.png,True,G19658,train,ec29991c2417b7838adfcd4f1be565a1,d026e31bc34fb44b
Viral Pneumonia-1048.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1048.png,Viral Pneumonia/masks/Viral Pneumonia-1048.png,True,G19659,train,83691967ffc6a1dd1ff4d2637fe94926,d46ce73be10f2807
Viral Pneumonia-1049.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1049.png,Viral Pneumonia/masks/Viral Pneumonia-1049.png,True,G19660,test,7e76d4c970fbe8527be18a71a90ba79a,91e5e6bae63a8628
Viral Pneumonia-105.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-105.png,Viral Pneumonia/masks/Viral Pneumonia-105.png,True,G19661,train,1117a4a60686065f169be0e114928c76,d4a8e31febaf2880
Viral Pneumonia-1050.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1050.png,Viral Pneumonia/masks/Viral Pneumonia-1050.png,True,G19662,val,ec4fc7ab1d5e2fabcf81c63bfb18d97a,d02faf0d0e4a1f56
Viral Pneumonia-1051.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1051.png,Viral Pneumonia/masks/Viral Pneumonia-1051.png,True,G19663,val,283ba8eea34c18ee9ab56cb659d51c59,d4aaef5555551614
Viral Pneumonia-1052.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1052.png,Viral Pneumonia/masks/Viral Pneumonia-1052.png,True,G19664,val,bea45adf9d89af380502199dcd6e230c,d46aaf1dc52dc143
Viral Pneumonia-1053.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1053.png,Viral Pneumonia/masks/Viral Pneumonia-1053.png,True,G19664,val,bea45adf9d89af380502199dcd6e230c,d46aaf1dc52dc143
Viral Pneumonia-1054.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1054.png,Viral Pneumonia/masks/Viral Pneumonia-1054.png,True,G19665,test,0f78de85e4cc2eef20016bbb98cb127b,d095e61ee50eb84e
Viral Pneumonia-1055.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1055.png,Viral Pneumonia/masks/Viral Pneumonia-1055.png,True,G19666,train,130af026653d1b69d7e05ca1a9b9b7b3,d46bef1bfa486810
Viral Pneumonia-1056.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1056.png,Viral Pneumonia/masks/Viral Pneumonia-1056.png,True,G19667,test,c1ca0a5efae745bf041c37476c9203c6,d82f531faa603ce1
Viral Pneumonia-1057.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1057.png,Viral Pneumonia/masks/Viral Pneumonia-1057.png,True,G19668,train,c1564fc4fb2e7aee84c2c559e7f667ae,90aee71ff04b38c2
Viral Pneumonia-1058.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1058.png,Viral Pneumonia/masks/Viral Pneumonia-1058.png,True,G19669,train,34d7c46419e23d685aedf09adef29493,903be65a3ed03935
Viral Pneumonia-1059.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1059.png,Viral Pneumonia/masks/Viral Pneumonia-1059.png,True,G19670,train,fa82e8c3bd0df3fa25da55edade364f2,c076e73aad692c85
Viral Pneumonia-106.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-106.png,Viral Pneumonia/masks/Viral Pneumonia-106.png,True,G19671,train,fecbd4ac6f961dee7cf928fc30b12631,c0a7e23bf8693cc2
Viral Pneumonia-1060.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1060.png,Viral Pneumonia/masks/Viral Pneumonia-1060.png,True,G19672,train,cd2d7a0412e0b3ddae28bde9957f9db7,9039e71ab2cf2e94
Viral Pneumonia-1061.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1061.png,Viral Pneumonia/masks/Viral Pneumonia-1061.png,True,G19673,train,9b73af695d2d1a1252fd3e3d29760d08,d03ea72e3ab50b38
Viral Pneumonia-1062.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1062.png,Viral Pneumonia/masks/Viral Pneumonia-1062.png,True,G19674,val,edbcb5c051f689c20f3fe714568fed4f,d476a32fa62d2c26
Viral Pneumonia-1063.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1063.png,Viral Pneumonia/masks/Viral Pneumonia-1063.png,True,G19675,val,f4b9f25e5a9779b37dd59229dbdde2db,d07c873aad293c4e
Viral Pneumonia-1064.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1064.png,Viral Pneumonia/masks/Viral Pneumonia-1064.png,True,G19676,train,1b0c7421b1df08999c8e957608220bb9,d030c71ac34fb4cf
Viral Pneumonia-1065.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1065.png,Viral Pneumonia/masks/Viral Pneumonia-1065.png,True,G19677,train,5732d5c42ea4c49ca0bae680062308d6,822ad25e3cb83d97
Viral Pneumonia-1066.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1066.png,Viral Pneumonia/masks/Viral Pneumonia-1066.png,True,G19678,train,fb5f2be2e7533554ecdcf92bb7a3d3de,80ade23af86938cf
Viral Pneumonia-1067.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1067.png,Viral Pneumonia/masks/Viral Pneumonia-1067.png,True,G19679,train,65d40f8293a3e4fba059f33a9d1e9be3,d42fe72fad09180b
Viral Pneumonia-1068.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1068.png,Viral Pneumonia/masks/Viral Pneumonia-1068.png,True,G19680,train,57aecbbe0b06c5236d84bafc53648eaf,d06de73ee30b8462
Viral Pneumonia-1069.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1069.png,Viral Pneumonia/masks/Viral Pneumonia-1069.png,True,G19681,test,64a8d8c1d1291eda560195e416558272,912fc61ae868b46f
Viral Pneumonia-107.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-107.png,Viral Pneumonia/masks/Viral Pneumonia-107.png,True,G19682,train,bf0f6c745c5a0b7e76d5a5ef95a61c1e,d074e24b97cb2d92
Viral Pneumonia-1070.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1070.png,Viral Pneumonia/masks/Viral Pneumonia-1070.png,True,G19683,train,d9db867c9cded06b23050f128c38e838,913ec65e94c63d94
Viral Pneumonia-1071.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1071.png,Viral Pneumonia/masks/Viral Pneumonia-1071.png,True,G19684,train,6ae4051beb822f6ba8a6e796aaed57fb,9029c61ede4c3df4
Viral Pneumonia-1072.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1072.png,Viral Pneumonia/masks/Viral Pneumonia-1072.png,True,G19685,val,32614d94a9a99cfed53be970d4ebd154,d42f830b074f6dc6
Viral Pneumonia-1073.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1073.png,Viral Pneumonia/masks/Viral Pneumonia-1073.png,True,G19686,test,cf1e2a9687e3293bb0107076a7ea17e3,d07ec61b9dcd2d80
Viral Pneumonia-1074.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1074.png,Viral Pneumonia/masks/Viral Pneumonia-1074.png,True,G19687,train,d236a896f8dac3ed8d8dee355c6f4ec8,d4afe70f55491d04
Viral Pneumonia-1075.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1075.png,Viral Pneumonia/masks/Viral Pneumonia-1075.png,True,G19688,train,5da07cda910b7ec1aff396264a681f64,d02bee1ede4b3c40
Viral Pneumonia-1076.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1076.png,Viral Pneumonia/masks/Viral Pneumonia-1076.png,True,G19689,train,440d8bb5b9b52054166062bba7b400c8,d060c75adb4f49c5
Viral Pneumonia-1077.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1077.png,Viral Pneumonia/masks/Viral Pneumonia-1077.png,True,G19690,train,77c005c8460410d228d6778dd228157d,c01ec21bad6d3ccb
Viral Pneumonia-1078.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1078.png,Viral Pneumonia/masks/Viral Pneumonia-1078.png,True,G19691,train,2cbb88f1a8138fc0f41344f054858090,946ce71eaf0b1b81
Viral Pneumonia-1079.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1079.png,Viral Pneumonia/masks/Viral Pneumonia-1079.png,True,G19692,train,0586f8becc0b2f3e10a485dd1eb74be7,d4ee870bc11f1917
Viral Pneumonia-108.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-108.png,Viral Pneumonia/masks/Viral Pneumonia-108.png,True,G19693,train,90f1c5d6946da13df42d6efc48f41d6c,d4efaf0f414c4c0e
Viral Pneumonia-1080.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1080.png,Viral Pneumonia/masks/Viral Pneumonia-1080.png,True,G19694,train,88a4f99c61b40be2289aa8f543aeb8df,812be60a38e97b7c
Viral Pneumonia-1081.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1081.png,Viral Pneumonia/masks/Viral Pneumonia-1081.png,True,G19695,train,c61acad7dd83e15b11a59c4e203254f3,d0eaa3adada43c54
Viral Pneumonia-1082.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1082.png,Viral Pneumonia/masks/Viral Pneumonia-1082.png,True,G19696,train,b3dca06468ebb621ace72b15e7e6c7ec,d06ec71d9da43c86
Viral Pneumonia-1083.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1083.png,Viral Pneumonia/masks/Viral Pneumonia-1083.png,True,G19697,train,e13e54b14ce6a0185738935d39e6bee3,d03de1435acb2f2c
Viral Pneumonia-1084.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1084.png,Viral Pneumonia/masks/Viral Pneumonia-1084.png,True,G19698,train,da164d732a26fb82672a9264558cf720,9428eb5594d21ed7
Viral Pneumonia-1085.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1085.png,Viral Pneumonia/masks/Viral Pneumonia-1085.png,True,G19699,train,a8539bca0dd8cfede5917577c1b453fd,d47ce30b872f3906
Viral Pneumonia-1086.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1086.png,Viral Pneumonia/masks/Viral Pneumonia-1086.png,True,G19700,val,ed0370ea08579ab5cc8a4fc86e5e49db,903be33ada5c3994
Viral Pneumonia-1087.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1087.png,Viral Pneumonia/masks/Viral Pneumonia-1087.png,True,G19701,train,963f0fb30de0b7dbf7d7ea014187061d,d9a4e21aea2bb24d
Viral Pneumonia-1088.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1088.png,Viral Pneumonia/masks/Viral Pneumonia-1088.png,True,G19702,val,dd2bad79f32dccd309be123e09f257a4,d074e73ab28b29a5
Viral Pneumonia-1089.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1089.png,Viral Pneumonia/masks/Viral Pneumonia-1089.png,True,G19703,test,fba997d51b7095e616b2f6c63d855a96,917ca67a8b3f0a83
Viral Pneumonia-109.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-109.png,Viral Pneumonia/masks/Viral Pneumonia-109.png,True,G19704,test,6c8b056bf31d8aaaeb49b08fc0118e9d,d47ce73fad2d2800
Viral Pneumonia-1090.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1090.png,Viral Pneumonia/masks/Viral Pneumonia-1090.png,True,G19705,val,85b0c0982c1199a0892d220534f5ab46,c07ae41ae7966b85
Viral Pneumonia-1091.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1091.png,Viral Pneumonia/masks/Viral Pneumonia-1091.png,True,G19706,test,4c12d40582af36cd7667dd52ce369428,911fe2ce9ac46e38
Viral Pneumonia-1092.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1092.png,Viral Pneumonia/masks/Viral Pneumonia-1092.png,True,G19707,val,63d0de931193b37c27f5d38ddf7f9b13,d127c31bc6693c63
Viral Pneumonia-1093.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1093.png,Viral Pneumonia/masks/Viral Pneumonia-1093.png,True,G19708,test,4d5841ebf5e69f8a3d4bff9b8a471647,d042c31fd52fb463
Viral Pneumonia-1094.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1094.png,Viral Pneumonia/masks/Viral Pneumonia-1094.png,True,G19709,train,1e5016b23c7983011f4f88cd1af44b3f,d46ac31bc92d3c87
Viral Pneumonia-1095.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1095.png,Viral Pneumonia/masks/Viral Pneumonia-1095.png,True,G19710,train,0bbfcb98156db156941992f5a4ea8505,8165c61ae33faa63
Viral Pneumonia-1096.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1096.png,Viral Pneumonia/masks/Viral Pneumonia-1096.png,True,G19711,train,bc972281188ce459b68d0f7cf11951ef,d075c61abc4bbe42
Viral Pneumonia-1097.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1097.png,Viral Pneumonia/masks/Viral Pneumonia-1097.png,True,G19712,train,b17fb524e841a55436aed133fd2d3123,913fc61aa87939c3
Viral Pneumonia-1098.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1098.png,Viral Pneumonia/masks/Viral Pneumonia-1098.png,True,G19713,train,3d20478331709f5d2e98793de3859c10,d06be75f500a3c47
Viral Pneumonia-1099.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1099.png,Viral Pneumonia/masks/Viral Pneumonia-1099.png,True,G19714,train,c072689f7155a310a4a52389314ab195,d0eea711555d6986
Viral Pneumonia-11.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-11.png,Viral Pneumonia/masks/Viral Pneumonia-11.png,True,G19715,train,f02e22c0b439ee59f125e6c885452ce1,9d2b875b4b2c2c36
Viral Pneumonia-110.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-110.png,Viral Pneumonia/masks/Viral Pneumonia-110.png,True,G19716,test,dba38b6a551aa5758df75950d429209b,d47c875b8b0b0997
Viral Pneumonia-1100.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1100.png,Viral Pneumonia/masks/Viral Pneumonia-1100.png,True,G19717,train,3994e662cedc0d1e0e2d1098c26e2673,d02fef5f5c481914
Viral Pneumonia-1101.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1101.png,Viral Pneumonia/masks/Viral Pneumonia-1101.png,True,G19718,train,b3bb696b13b4434bd010c9048499d238,907f07180d5d5f47
Viral Pneumonia-1102.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1102.png,Viral Pneumonia/masks/Viral Pneumonia-1102.png,True,G19719,train,bb4f3abe6cced06d4e37e3a9de4a81b4,d460a33be34bc61e
Viral Pneumonia-1103.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1103.png,Viral Pneumonia/masks/Viral Pneumonia-1103.png,True,G19720,train,3dced5beef2d667cd16f3d564363b31e,d42be33bea4a1cc1
Viral Pneumonia-1104.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1104.png,Viral Pneumonia/masks/Viral Pneumonia-1104.png,True,G19721,train,b7092ccd08e334dfdb68d03b29d24299,9474875bc51be13c
Viral Pneumonia-1105.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1105.png,Viral Pneumonia/masks/Viral Pneumonia-1105.png,True,G19722,train,0feeb9954bead5e2a5c8a8564505b3cd,d0a5e73bb04b3d50
Viral Pneumonia-1106.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1106.png,Viral Pneumonia/masks/Viral Pneumonia-1106.png,True,G19723,train,6cabee390e758e53a43faed2c6ff6a0b,9527e71a92cab8c6
Viral Pneumonia-1107.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1107.png,Viral Pneumonia/masks/Viral Pneumonia-1107.png,True,G19724,val,c0acbc9d54ccda37e8a447764722c0d0,d078c739ec3c3c8c
Viral Pneumonia-1108.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1108.png,Viral Pneumonia/masks/Viral Pneumonia-1108.png,True,G19725,val,60e2222921fe47a5aca60a997d7de974,81a4e23ee26f78ca
Viral Pneumonia-1109.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1109.png,Viral Pneumonia/masks/Viral Pneumonia-1109.png,True,G19726,test,032e2635dda11d6ec2511372bdc84e8d,91ae8e389578b45e
Viral Pneumonia-111.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-111.png,Viral Pneumonia/masks/Viral Pneumonia-111.png,True,G19727,train,ad6679f9aa78307a6ccd0da18c0912da,917ca63aa93d383c
Viral Pneumonia-1110.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1110.png,Viral Pneumonia/masks/Viral Pneumonia-1110.png,True,G19728,train,2c6077d54f2144cd88feb5945430f449,d07de31aed4d6884
Viral Pneumonia-1111.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1111.png,Viral Pneumonia/masks/Viral Pneumonia-1111.png,True,G19729,train,6b19a4a1b5e9f2577a7f5843384936dc,9578ae3a2b0d390f
Viral Pneumonia-1112.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1112.png,Viral Pneumonia/masks/Viral Pneumonia-1112.png,True,G19730,train,3b24ec14e571bbfc69594c7ddf462928,916fc638e939a46a
Viral Pneumonia-1113.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1113.png,Viral Pneumonia/masks/Viral Pneumonia-1113.png,True,G19731,train,e05c3e1150ab30919350dadd56f7c55b,c0bff75a104b1e4a
Viral Pneumonia-1114.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1114.png,Viral Pneumonia/masks/Viral Pneumonia-1114.png,True,G19732,train,430827b8c048f41a5283b19a6eccf3c9,85ae2f6aeae86941
Viral Pneumonia-1115.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1115.png,Viral Pneumonia/masks/Viral Pneumonia-1115.png,True,G19733,train,5b7cc175df314d0baab8ac5e07abbf7f,c22feb2d91455553
Viral Pneumonia-1116.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1116.png,Viral Pneumonia/masks/Viral Pneumonia-1116.png,True,G19734,train,aabf6e770d4eccdd079aafc84251b03e,913e6a5c544f3d85
Viral Pneumonia-1117.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1117.png,Viral Pneumonia/masks/Viral Pneumonia-1117.png,True,G19735,train,86c1e56306d5fdeced78060bebb2f7cf,906de73ea20bba43
Viral Pneumonia-1118.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1118.png,Viral Pneumonia/masks/Viral Pneumonia-1118.png,True,G19736,train,e21260922d49c5f10302cab9bd72aabb,80aa565c56595d5f
Viral Pneumonia-1119.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1119.png,Viral Pneumonia/masks/Viral Pneumonia-1119.png,True,G19737,train,720b02e22263c8998217953a34083155,d00ba3abbee82b1c
Viral Pneumonia-112.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-112.png,Viral Pneumonia/masks/Viral Pneumonia-112.png,True,G19738,val,a97073bbc2646bbe10e5983b1db4a36e,d06ca73abf0b290e
Viral Pneumonia-1120.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1120.png,Viral Pneumonia/masks/Viral Pneumonia-1120.png,True,G19739,train,c27113f373a810f46cb2ffcd6e22b3ba,917fe61a3949390d
Viral Pneumonia-1121.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1121.png,Viral Pneumonia/masks/Viral Pneumonia-1121.png,True,G19740,test,f9c32b320614399b2dd3bea417dcb278,d13dc67a944f34c2
Viral Pneumonia-1122.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1122.png,Viral Pneumonia/masks/Viral Pneumonia-1122.png,True,G19741,test,2086b8a81845843091b7dbeb88b2c937,917ec71a1c4939dc
Viral Pneumonia-1123.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1123.png,Viral Pneumonia/masks/Viral Pneumonia-1123.png,True,G19742,val,84d5022193ba275fa4746f4e1e3c22d6,903de55fb4c21ac6
Viral Pneumonia-1124.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1124.png,Viral Pneumonia/masks/Viral Pneumonia-1124.png,True,G19743,train,12f8c1258168c41f210d3b0f6439e769,c02fc30f9cc94d9e
Viral Pneumonia-1125.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1125.png,Viral Pneumonia/masks/Viral Pneumonia-1125.png,True,G19744,train,af53c350a5016a9ec1d59edf00d88e09,956e2f3b0f09091f
Viral Pneumonia-1126.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1126.png,Viral Pneumonia/masks/Viral Pneumonia-1126.png,True,G19745,train,480e044760887ed081255eca7de9f914,9479c71ae9cb6894
Viral Pneumonia-1127.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1127.png,Viral Pneumonia/masks/Viral Pneumonia-1127.png,True,G19746,test,2a59dac26a0b363eb0f7f77bdadadaa0,d434c31bfdcc3894
Viral Pneumonia-1128.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1128.png,Viral Pneumonia/masks/Viral Pneumonia-1128.png,True,G19747,train,e282125daca74faa677267d8fc21222c,d07ae51f6a971609
Viral Pneumonia-1129.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1129.png,Viral Pneumonia/masks/Viral Pneumonia-1129.png,True,G19748,val,74128dc10b22773901f640f2ebb87757,d007e70fb4c83db8
Viral Pneumonia-113.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-113.png,Viral Pneumonia/masks/Viral Pneumonia-113.png,True,G19749,train,9ce2bf85f75a197fd09d040a846907f2,d06cef3ead482c4c
Viral Pneumonia-1130.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1130.png,Viral Pneumonia/masks/Viral Pneumonia-1130.png,True,G19750,train,6c4b9c0575de340097ffed9c1a1ac1d2,d47e834b83176995
Viral Pneumonia-1131.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1131.png,Viral Pneumonia/masks/Viral Pneumonia-1131.png,True,G19751,train,735e842e61480d9792c93ae9758c4a25,d42ec31b994a5cd6
Viral Pneumonia-1132.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1132.png,Viral Pneumonia/masks/Viral Pneumonia-1132.png,True,G19752,val,6f4a8b842d1ae17e7b98512eefaba069,d13bea3a5c4d1d41
Viral Pneumonia-1133.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1133.png,Viral Pneumonia/masks/Viral Pneumonia-1133.png,True,G19753,train,2f5df19940f5550eb6461793e1edb055,94abebbef8411c44
Viral Pneumonia-1134.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1134.png,Viral Pneumonia/masks/Viral Pneumonia-1134.png,True,G19754,train,cc051104118baf57080fee3bf00b007a,d02bff565e421a1a
Viral Pneumonia-1135.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1135.png,Viral Pneumonia/masks/Viral Pneumonia-1135.png,True,G19755,train,206c3289c013d14afeb2d58720f7067e,d064c65acf3d853c
Viral Pneumonia-1136.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1136.png,Viral Pneumonia/masks/Viral Pneumonia-1136.png,True,G19756,train,13fe7ee7060ad976dbac47bf00b0c7ea,d135c61ac66e3cc6
Viral Pneumonia-1137.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1137.png,Viral Pneumonia/masks/Viral Pneumonia-1137.png,True,G19757,test,ef4d3b16e161d0b22b08adb1fdd0ecc8,d12ae25ed64f1dc0
Viral Pneumonia-1138.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1138.png,Viral Pneumonia/masks/Viral Pneumonia-1138.png,True,G19758,test,4542416720dc2e3272305854f1e1445a,d12fc34ac56978e4
Viral Pneumonia-1139.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1139.png,Viral Pneumonia/masks/Viral Pneumonia-1139.png,True,G19759,train,ab335124ba23e05371fe6f652859ed67,d02ac25f5f6a1a4b
Viral Pneumonia-114.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-114.png,Viral Pneumonia/masks/Viral Pneumonia-114.png,True,G19760,train,dfea645948eeea6354fe7befd07aeb4b,9429e38bea5e6a45
Viral Pneumonia-1140.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1140.png,Viral Pneumonia/masks/Viral Pneumonia-1140.png,True,G19761,train,bcae88cf6e36ea5c5bab07e0e9f06b83,d06ae23be9792869
Viral Pneumonia-1141.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1141.png,Viral Pneumonia/masks/Viral Pneumonia-1141.png,True,G19762,train,3e3ed0faed0180efee64319027640b4c,d03bd65e1a4a391e
Viral Pneumonia-1142.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1142.png,Viral Pneumonia/masks/Viral Pneumonia-1142.png,True,G19763,train,9b324b89168b15c93f2c2610e77cd301,d07feb1e0a4b1847
Viral Pneumonia-1143.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1143.png,Viral Pneumonia/masks/Viral Pneumonia-1143.png,True,G19764,train,f25bd41d8dbcbab3bf7fa8456d430d7e,d17de63ae9393801
Viral Pneumonia-1144.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1144.png,Viral Pneumonia/masks/Viral Pneumonia-1144.png,True,G19765,val,52db57a5bc38ce8b1ee5506648d0d49d,c06ec74badec2d84
Viral Pneumonia-1145.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1145.png,Viral Pneumonia/masks/Viral Pneumonia-1145.png,True,G19766,val,719bf68a1afd75e1a959d43985c7ee21,d42eaf5b1d534914
Viral Pneumonia-1146.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1146.png,Viral Pneumonia/masks/Viral Pneumonia-1146.png,True,G19767,test,d879e6ed33609972891f5c730d09ac81,d43a525f1e283cb9
Viral Pneumonia-1147.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1147.png,Viral Pneumonia/masks/Viral Pneumonia-1147.png,True,G19768,train,1eb8925069e6034b079559ea93eb9b55,d07de73ea30e3a02
Viral Pneumonia-1148.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1148.png,Viral Pneumonia/masks/Viral Pneumonia-1148.png,True,G19769,train,87be28cb3d1e8c5897afef828989f646,d03de51faec22b82
Viral Pneumonia-1149.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1149.png,Viral Pneumonia/masks/Viral Pneumonia-1149.png,True,G19770,train,3f8fc1f35388a11e58ecf2bb7fc1df30,c062d338e56de4e6
Viral Pneumonia-115.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-115.png,Viral Pneumonia/masks/Viral Pneumonia-115.png,True,G19771,train,1ef1908a02416a281f068ab3b0fcab25,d035c20f9acf3dc2
Viral Pneumonia-1150.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1150.png,Viral Pneumonia/masks/Viral Pneumonia-1150.png,True,G19772,train,f8964e10ab833f914f4c67d0b5baceb3,9568c71abdcd2989
Viral Pneumonia-1151.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1151.png,Viral Pneumonia/masks/Viral Pneumonia-1151.png,True,G19773,train,1c1257932ed38c920d3c5a8bb785cd07,942f471f4b5b1c46
Viral Pneumonia-1152.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1152.png,Viral Pneumonia/masks/Viral Pneumonia-1152.png,True,G19774,train,dbb5b7b256e55a86491ae9eefab8750a,947e831b431f4c4f
Viral Pneumonia-1153.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1153.png,Viral Pneumonia/masks/Viral Pneumonia-1153.png,True,G19775,train,56dcd93afc8afaf82ba32a641a66d37e,d03ce69bbf890b0c
Viral Pneumonia-1154.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1154.png,Viral Pneumonia/masks/Viral Pneumonia-1154.png,True,G19776,train,9883f77b06d756587339540cd28c3014,d454e44f7a874e38
Viral Pneumonia-1155.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1155.png,Viral Pneumonia/masks/Viral Pneumonia-1155.png,True,G19777,train,3d80495e0e5595c03a81ddf7853635b0,9420afb0eb38eab9
Viral Pneumonia-1156.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1156.png,Viral Pneumonia/masks/Viral Pneumonia-1156.png,True,G19778,train,1cba22ebf1bcf3455cccb9ee5a7d1a7f,c0bf470a0a4b7d5d
Viral Pneumonia-1157.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1157.png,Viral Pneumonia/masks/Viral Pneumonia-1157.png,True,G19779,train,dc0b899b01650a0007706e1160281139,9117c61eb8ca7db0
Viral Pneumonia-1158.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1158.png,Viral Pneumonia/masks/Viral Pneumonia-1158.png,True,G19780,train,622f46fa1e6a43a8f0807fb45ee26609,9168b67a8f174b94
Viral Pneumonia-1159.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1159.png,Viral Pneumonia/masks/Viral Pneumonia-1159.png,True,G19781,val,3531d9a6983985d99a6d63837343831d,81ebea3ae9593849
Viral Pneumonia-116.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-116.png,Viral Pneumonia/masks/Viral Pneumonia-116.png,True,G19782,train,bf977323ed29f57decce47e417a3ded8,d07ed71f851c3c06
Viral Pneumonia-1160.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1160.png,Viral Pneumonia/masks/Viral Pneumonia-1160.png,True,G19783,val,682974d205bec300eaf8acc4425dc289,c02ffa5ed24b5648
Viral Pneumonia-1161.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1161.png,Viral Pneumonia/masks/Viral Pneumonia-1161.png,True,G19784,val,e5b39a02a3dc482e316d8472f8141783,cae0d679cd386592
Viral Pneumonia-1162.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1162.png,Viral Pneumonia/masks/Viral Pneumonia-1162.png,True,G19785,test,0d9eeda47ef24e94fd53378e378a081b,9165c61ac23af64f
Viral Pneumonia-1163.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1163.png,Viral Pneumonia/masks/Viral Pneumonia-1163.png,True,G19786,test,48bf69dbeea9703e8fc6eb516afd4eac,913fe3bfb8c80a18
Viral Pneumonia-1164.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1164.png,Viral Pneumonia/masks/Viral Pneumonia-1164.png,True,G19787,test,cd64f59b8b0fadc508ff0ce084ea35ca,913fc75ebcc12990
Viral Pneumonia-1165.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1165.png,Viral Pneumonia/masks/Viral Pneumonia-1165.png,True,G19788,train,ccd3a1776fbce135cbdfd088bfca49dc,d07ac33fc52db442
Viral Pneumonia-1166.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1166.png,Viral Pneumonia/masks/Viral Pneumonia-1166.png,True,G19789,train,9db932bb509294e901557b2b4e4bc110,d046defd850cc729
Viral Pneumonia-1167.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1167.png,Viral Pneumonia/masks/Viral Pneumonia-1167.png,True,G19790,train,46c9012aba30d93dba6c4dd9a425d32b,d73c6669794a6912
Viral Pneumonia-1168.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1168.png,Viral Pneumonia/masks/Viral Pneumonia-1168.png,True,G19791,train,81c731932607c6be48228929e5911d32,d068c61a99b43df6
Viral Pneumonia-1169.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1169.png,Viral Pneumonia/masks/Viral Pneumonia-1169.png,True,G19792,train,42c06b53deb2ad878b368740ee3ac046,d035c71b9dc93dc0
Viral Pneumonia-117.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-117.png,Viral Pneumonia/masks/Viral Pneumonia-117.png,True,G19793,train,84ffdd23e87c71c2da6a75b61cd82d6f,902be7de1cf43131
Viral Pneumonia-1170.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1170.png,Viral Pneumonia/masks/Viral Pneumonia-1170.png,True,G19794,train,b17368121f31411c66f5ba0fe35ebb16,d474e31be91f2845
Viral Pneumonia-1171.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1171.png,Viral Pneumonia/masks/Viral Pneumonia-1171.png,True,G19795,train,9c7ac8e8f7249d1253f1608d7f56374d,906fc73ec37c7042
Viral Pneumonia-1172.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1172.png,Viral Pneumonia/masks/Viral Pneumonia-1172.png,True,G19796,train,df2e74cecf1238397981d8c5e2d6b637,9516c71aadcc38c7
Viral Pneumonia-1173.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1173.png,Viral Pneumonia/masks/Viral Pneumonia-1173.png,True,G19797,val,64d67aa6f04b7226ea5dc5ce41d078c5,d4aee33ba70b3882
Viral Pneumonia-1174.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1174.png,Viral Pneumonia/masks/Viral Pneumonia-1174.png,True,G19798,train,da7ce770d56161cf0a8af2bbd0d642a8,c06ae35f854d6cc6
Viral Pneumonia-1175.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1175.png,Viral Pneumonia/masks/Viral Pneumonia-1175.png,True,G19799,train,9d607e83bc05b3234a3f08c90e088205,d0eaa73fa50f2807
Viral Pneumonia-1176.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1176.png,Viral Pneumonia/masks/Viral Pneumonia-1176.png,True,G19800,test,98323c0fd73fceea32a10565a49fd266,9568ae3a1d1f194d
Viral Pneumonia-1177.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1177.png,Viral Pneumonia/masks/Viral Pneumonia-1177.png,True,G19801,train,a426f150586899af36cdd10ecb9a2042,d07ec71facc528c4
Viral Pneumonia-1178.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1178.png,Viral Pneumonia/masks/Viral Pneumonia-1178.png,True,G19802,test,4ece98a4d1081e523a9ef97a0a9ba78b,d476e23bc50995c3
Viral Pneumonia-1179.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1179.png,Viral Pneumonia/masks/Viral Pneumonia-1179.png,True,G19803,train,ebbed078f0f9bd96314653fdfe5e2dbf,902ae32e5ec03b7e
Viral Pneumonia-118.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-118.png,Viral Pneumonia/masks/Viral Pneumonia-118.png,True,G19793,train,84ffdd23e87c71c2da6a75b61cd82d6f,902be7de1cf43131
Viral Pneumonia-1180.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1180.png,Viral Pneumonia/masks/Viral Pneumonia-1180.png,True,G19804,train,198378ac79e61ed28030457ad334aeb3,90a1e31adeca9ec6
Viral Pneumonia-1181.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1181.png,Viral Pneumonia/masks/Viral Pneumonia-1181.png,True,G19805,val,60f07738268a3d002c9dc86773c1c885,912be23a785a5a79
Viral Pneumonia-1182.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1182.png,Viral Pneumonia/masks/Viral Pneumonia-1182.png,True,G19806,val,894306ec38bde4da279591aea90c5398,d02de31fd1491de8
Viral Pneumonia-1183.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1183.png,Viral Pneumonia/masks/Viral Pneumonia-1183.png,True,G19807,test,684216cf66a1e2e7752c83bcb0d8e11e,957eae1e080a1d5f
Viral Pneumonia-1184.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1184.png,Viral Pneumonia/masks/Viral Pneumonia-1184.png,True,G19808,train,9e3c9d97625d806df450bab73d140643,d4f6a75a910f2986
Viral Pneumonia-1185.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1185.png,Viral Pneumonia/masks/Viral Pneumonia-1185.png,True,G19809,train,09fa1709019bd582800a0c00a9dcb2fd,912be31ae8497979
Viral Pneumonia-1186.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1186.png,Viral Pneumonia/masks/Viral Pneumonia-1186.png,True,G19810,train,0927eca3c98faad0a60ef334d79d0fcf,9076865ac53d49f6
Viral Pneumonia-1187.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1187.png,Viral Pneumonia/masks/Viral Pneumonia-1187.png,True,G19811,train,bfb2c70ebda831e30a461d547ff0bdce,9560c31e831be33f
Viral Pneumonia-1188.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1188.png,Viral Pneumonia/masks/Viral Pneumonia-1188.png,True,G19812,train,d5fb4e907e97845c0ab01c2a878d508d,d0ffef1a00491c4f
Viral Pneumonia-1189.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1189.png,Viral Pneumonia/masks/Viral Pneumonia-1189.png,True,G19813,train,13b4c3448a4c27a8f90ce2807eb8b4f2,d0fee70d80af2ca2
Viral Pneumonia-119.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-119.png,Viral Pneumonia/masks/Viral Pneumonia-119.png,True,G19814,train,eb65d6f32c27f05e981fa9a878c41ca5,913fe01eb8c39b31
Viral Pneumonia-1190.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1190.png,Viral Pneumonia/masks/Viral Pneumonia-1190.png,True,G19815,train,281ab6006aea3445ac892fb0c0eb6453,9175e71ae04f38c6
Viral Pneumonia-1191.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1191.png,Viral Pneumonia/masks/Viral Pneumonia-1191.png,True,G19816,train,c762d4cf69adc994751c39e4904c341b,9035e71eee5a38c2
Viral Pneumonia-1192.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1192.png,Viral Pneumonia/masks/Viral Pneumonia-1192.png,True,G19817,train,06d74e7dc739111eb8db513375e41934,d02ee71aad0b2dca
Viral Pneumonia-1193.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1193.png,Viral Pneumonia/masks/Viral Pneumonia-1193.png,True,G19818,train,41dcb844ba4418b5b67d3bbeefb44cbb,9568c73ec28b19c7
Viral Pneumonia-1194.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1194.png,Viral Pneumonia/masks/Viral Pneumonia-1194.png,True,G19819,train,fd9bb61a9df2f0b9874521b4b238afda,d47e871f871d011e
Viral Pneumonia-1195.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1195.png,Viral Pneumonia/masks/Viral Pneumonia-1195.png,True,G19820,val,caa631b80421a88516f2362d9376813c,94a9e33ee80b784b
Viral Pneumonia-1196.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1196.png,Viral Pneumonia/masks/Viral Pneumonia-1196.png,True,G19821,train,fbc58e2aa021b06a8697410d7420b0af,9525e31bbbcf28c0
Viral Pneumonia-1197.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1197.png,Viral Pneumonia/masks/Viral Pneumonia-1197.png,True,G19822,test,90fcf36057c9d6e119638be644ae69c7,975a8618318d797b
Viral Pneumonia-1198.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1198.png,Viral Pneumonia/masks/Viral Pneumonia-1198.png,True,G19823,train,e76fe1e5e3df94a876d10396b2a68a12,911fc63ab94b6992
Viral Pneumonia-1199.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1199.png,Viral Pneumonia/masks/Viral Pneumonia-1199.png,True,G19824,train,f0957fd958ab6df4a663a68fb3839d24,d025c70ebe4b3d92
Viral Pneumonia-12.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-12.png,Viral Pneumonia/masks/Viral Pneumonia-12.png,True,G19825,val,0c4a0ff9ccc0e121e73b96ef6f9d0c74,d03ece3b18691d5c
Viral Pneumonia-120.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-120.png,Viral Pneumonia/masks/Viral Pneumonia-120.png,True,G19826,val,eb5301050632045f9ab8a6fe1676a280,d729a2321c193b3f
Viral Pneumonia-1200.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1200.png,Viral Pneumonia/masks/Viral Pneumonia-1200.png,True,G19827,test,e221014e9db257c3549c26c6406caa36,d02dc21abf693c69
Viral Pneumonia-1201.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1201.png,Viral Pneumonia/masks/Viral Pneumonia-1201.png,True,G19828,test,f72b5b99569f71d1e17646b7ab3826d3,d47d875a9b4b3a10
Viral Pneumonia-1202.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1202.png,Viral Pneumonia/masks/Viral Pneumonia-1202.png,True,G19829,train,97a9589cbf3f67cdd615bd23a2fc8bf8,d17ec67ac80e3893
Viral Pneumonia-1203.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1203.png,Viral Pneumonia/masks/Viral Pneumonia-1203.png,True,G19830,train,ba5c611fcb757f96299f4c7f71de8018,912fe65c1c4a395d
Viral Pneumonia-1204.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1204.png,Viral Pneumonia/masks/Viral Pneumonia-1204.png,True,G19831,train,06ce72d4ef75e26683d25a054875cc94,d02faf1e415d544e
Viral Pneumonia-1205.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1205.png,Viral Pneumonia/masks/Viral Pneumonia-1205.png,True,G19832,train,08d3ce571a98c0647160f680402bce44,d02da71bcb0b3cca
Viral Pneumonia-1206.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1206.png,Viral Pneumonia/masks/Viral Pneumonia-1206.png,True,G19833,train,5119d23c7e4f65a4da31342c4f7e49f8,d87dd71b827ac04a
Viral Pneumonia-1207.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1207.png,Viral Pneumonia/masks/Viral Pneumonia-1207.png,True,G19834,val,59c2413411bb860696c303f17b75ba84,d020e39bcb4bb24f
Viral Pneumonia-1208.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1208.png,Viral Pneumonia/masks/Viral Pneumonia-1208.png,True,G19835,train,582ed48796c1e10d24cdc49de441ac5f,81eaae1a875f115e
Viral Pneumonia-1209.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1209.png,Viral Pneumonia/masks/Viral Pneumonia-1209.png,True,G19836,test,689409551aec23a0110b236568dfd534,d074e71a930bb94b
Viral Pneumonia-121.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-121.png,Viral Pneumonia/masks/Viral Pneumonia-121.png,True,G19837,train,bafdc147dd6acc47aba1ac85d04621f5,902ad25ade595b5a
Viral Pneumonia-1210.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1210.png,Viral Pneumonia/masks/Viral Pneumonia-1210.png,True,G19838,train,744f3bb34f2f41a779d5ca32f09f4c17,d440c31fd31bbd43
Viral Pneumonia-1211.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1211.png,Viral Pneumonia/masks/Viral Pneumonia-1211.png,True,G19839,train,ce3e0c9e184260b6ed2ee4df5af7035b,953ff65612405976
Viral Pneumonia-1212.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1212.png,Viral Pneumonia/masks/Viral Pneumonia-1212.png,True,G19840,train,38d773f301f55581533fb705c7f70cde,d42aab0b990d6cf6
Viral Pneumonia-1213.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1213.png,Viral Pneumonia/masks/Viral Pneumonia-1213.png,True,G19841,train,92bc5ee96e3f5f652702ef16d338b5cd,d029e36fbeb06115
Viral Pneumonia-1214.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1214.png,Viral Pneumonia/masks/Viral Pneumonia-1214.png,True,G19842,train,ccb33bba6a828b00a665e48a5dbd97e7,d03fd2d07869296b
Viral Pneumonia-1215.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1215.png,Viral Pneumonia/masks/Viral Pneumonia-1215.png,True,G19843,train,07cbf013066e06671f37f45f28058e59,947be33ab20a399c
Viral Pneumonia-1216.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1216.png,Viral Pneumonia/masks/Viral Pneumonia-1216.png,True,G19844,train,492258412fe91c5d64839cea68e61758,d06ae73f892d2d06
Viral Pneumonia-1217.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1217.png,Viral Pneumonia/masks/Viral Pneumonia-1217.png,True,G19845,train,c66a13eaa0a4b23efb72681117674de6,c13f7a1a4069746f
Viral Pneumonia-1218.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1218.png,Viral Pneumonia/masks/Viral Pneumonia-1218.png,True,G19846,val,daeeed139d6c308c9cfd713dc2aecca6,dd3fc34f155b1880
Viral Pneumonia-1219.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1219.png,Viral Pneumonia/masks/Viral Pneumonia-1219.png,True,G19847,train,5a0c766714509c8ed6c799e26b19f966,903fd61c1535b497
Viral Pneumonia-122.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-122.png,Viral Pneumonia/masks/Viral Pneumonia-122.png,True,G19848,val,b9f2b375daee0f1b5e4746f943f1a95a,9065c71eb44b3dc5
Viral Pneumonia-1220.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1220.png,Viral Pneumonia/masks/Viral Pneumonia-1220.png,True,G19849,test,0ecdef89aa80b20e69281ab41a34bad5,d97fc61e420b304f
Viral Pneumonia-1221.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1221.png,Viral Pneumonia/masks/Viral Pneumonia-1221.png,True,G19850,test,bd030018047f5c991507c0d6d15e0a7c,9074c71a93cf3996
Viral Pneumonia-1222.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1222.png,Viral Pneumonia/masks/Viral Pneumonia-1222.png,True,G19851,train,5d0ecb4f20aa8ccae57f1735fcb41b2d,816ce65e17471ad6
Viral Pneumonia-1223.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1223.png,Viral Pneumonia/masks/Viral Pneumonia-1223.png,True,G19852,train,9cb0ffef7dbf7e54d9cc0fe25b97db56,902fd71b830f384f
Viral Pneumonia-1224.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1224.png,Viral Pneumonia/masks/Viral Pneumonia-1224.png,True,G19853,train,eb2fbb7052ac74d07bbc9fddc0ecf6e2,9135c61bd56bb04b
Viral Pneumonia-1225.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1225.png,Viral Pneumonia/masks/Viral Pneumonia-1225.png,True,G19854,train,773920733c9bffef93e59263daed1008,91659339ed39c11b
Viral Pneumonia-1226.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1226.png,Viral Pneumonia/masks/Viral Pneumonia-1226.png,True,G19855,train,434c6a7dea36c537adf7aabb27420be3,d02fe35f5c421e58
Viral Pneumonia-1227.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1227.png,Viral Pneumonia/masks/Viral Pneumonia-1227.png,True,G19856,train,17c3dddaa6142e6371ad5f454d112990,c06c64dcda9bc34b
Viral Pneumonia-1228.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1228.png,Viral Pneumonia/masks/Viral Pneumonia-1228.png,True,G19857,train,065f119af02af44980aad4b952b659cd,d529f01e925a3d6a
Viral Pneumonia-1229.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1229.png,Viral Pneumonia/masks/Viral Pneumonia-1229.png,True,G19858,train,636d69cf83e5f19ebd6a8f23e64ee20b,c22fa419994b3f3c
Viral Pneumonia-123.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-123.png,Viral Pneumonia/masks/Viral Pneumonia-123.png,True,G19859,train,e3fe06fc00e32b93a0fb086a0f27dff8,d466965b85398b39
Viral Pneumonia-1230.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1230.png,Viral Pneumonia/masks/Viral Pneumonia-1230.png,True,G19860,train,bf029683832c6f2386192fdc102e07c7,d73b005ada1a7ee0
Viral Pneumonia-1231.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1231.png,Viral Pneumonia/masks/Viral Pneumonia-1231.png,True,G19861,val,1b63f97b26eebe723c5f9ac6ac60f8d5,903be72fbe421d18
Viral Pneumonia-1232.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1232.png,Viral Pneumonia/masks/Viral Pneumonia-1232.png,True,G19862,train,2ac14ee6281d9bf08e5d3f8e8bae7016,903be57cf82a38a9
Viral Pneumonia-1233.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1233.png,Viral Pneumonia/masks/Viral Pneumonia-1233.png,True,G19863,val,280f9f797a2e9f1a3272caf10269221a,9029e33ed6c23e5a
Viral Pneumonia-1234.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1234.png,Viral Pneumonia/masks/Viral Pneumonia-1234.png,True,G19864,train,16d77ad0b1c15d8f3c2dae0827ae1577,80eaef589719171e
Viral Pneumonia-1235.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1235.png,Viral Pneumonia/masks/Viral Pneumonia-1235.png,True,G19865,val,c9801a56db5f6f1e3b4c1db231003150,913fce1e5a1f184a
Viral Pneumonia-1236.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1236.png,Viral Pneumonia/masks/Viral Pneumonia-1236.png,True,G19866,test,a7421fb346d56afcd279f5082aa96239,d474e73bab0b2183
Viral Pneumonia-1237.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1237.png,Viral Pneumonia/masks/Viral Pneumonia-1237.png,True,G19867,test,09940f4e0847ce779d123518f1ab71e1,9125c23aff6a3c64
Viral Pneumonia-1238.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1238.png,Viral Pneumonia/masks/Viral Pneumonia-1238.png,True,G19868,train,79f9ff7a9bc07d9a87fb412b472ab24b,902ac71e5e525b73
Viral Pneumonia-1239.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1239.png,Viral Pneumonia/masks/Viral Pneumonia-1239.png,True,G19869,train,ac2156c82692c1766260729c7ca5f62c,c82b8b0fc56b26e3
Viral Pneumonia-124.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-124.png,Viral Pneumonia/masks/Viral Pneumonia-124.png,True,G19870,test,10c672f8d87c64ea6d72944f1604e0b9,9470c75ab10f7897
Viral Pneumonia-1240.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1240.png,Viral Pneumonia/masks/Viral Pneumonia-1240.png,True,G19871,train,a109d74dbc4c433afc7d23a049356fcf,d13fc65a18c65939
Viral Pneumonia-1241.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1241.png,Viral Pneumonia/masks/Viral Pneumonia-1241.png,True,G19872,train,f7d04be7b6c607e637b5ccc21a17b689,912be7563e41693c
Viral Pneumonia-1242.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1242.png,Viral Pneumonia/masks/Viral Pneumonia-1242.png,True,G19873,train,a0a475103be2fcfe7208d3a0bad161c2,d03f4e5a1dd98998
Viral Pneumonia-1243.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1243.png,Viral Pneumonia/masks/Viral Pneumonia-1243.png,True,G19874,train,1b2b26b550b5c00846e37a0c9cfe28ef,902ae75bd54b79c0
Viral Pneumonia-1244.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1244.png,Viral Pneumonia/masks/Viral Pneumonia-1244.png,True,G19875,train,93d29accbd18caa59d7f7b4248101c64,90eca75b875aa41e
Viral Pneumonia-1245.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1245.png,Viral Pneumonia/masks/Viral Pneumonia-1245.png,True,G19876,train,aa69068e61074b2ec43c8167722b2b01,903fde5e560a1c4b
Viral Pneumonia-1246.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1246.png,Viral Pneumonia/masks/Viral Pneumonia-1246.png,True,G19877,train,990ebddf615cf854f43f9773b4f87f56,9074c61ac74f29cf
Viral Pneumonia-1247.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1247.png,Viral Pneumonia/masks/Viral Pneumonia-1247.png,True,G19878,train,7375b2ea3a1b820ea600f3877c6d9b61,d47e871b8b1d083e
Viral Pneumonia-1248.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1248.png,Viral Pneumonia/masks/Viral Pneumonia-1248.png,True,G19879,train,839177931fcdf7fb919074f97e307713,9129e61ace6b3c69
Viral Pneumonia-1249.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1249.png,Viral Pneumonia/masks/Viral Pneumonia-1249.png,True,G19880,test,750d78e6efe384a2e3997a15ab274a42,882dd74a5d6b7d02
Viral Pneumonia-125.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-125.png,Viral Pneumonia/masks/Viral Pneumonia-125.png,True,G19881,test,46d8935690db37a0e4cd22af83142189,d0faf73f910d0990
Viral Pneumonia-1250.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1250.png,Viral Pneumonia/masks/Viral Pneumonia-1250.png,True,G19882,train,30b0f58c5b660eede1bb4bdbe7dc27d4,c46f272d856c943e
Viral Pneumonia-1251.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1251.png,Viral Pneumonia/masks/Viral Pneumonia-1251.png,True,G19883,train,a3036749b1a0729222f21d4eab04f10e,d03f0f1b0d49346f
Viral Pneumonia-1252.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1252.png,Viral Pneumonia/masks/Viral Pneumonia-1252.png,True,G19884,train,880fc06ff4c1efe146e43932a8de1327,803f535b973cb4b0
Viral Pneumonia-1253.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1253.png,Viral Pneumonia/masks/Viral Pneumonia-1253.png,True,G19885,val,182ee17e33a6f6270e644a689dc60883,d03fff1b5d3a1400
Viral Pneumonia-1254.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1254.png,Viral Pneumonia/masks/Viral Pneumonia-1254.png,True,G19886,val,38981b4a9c1db609ae57b17052b57231,90aae35fc55d444d
Viral Pneumonia-1255.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1255.png,Viral Pneumonia/masks/Viral Pneumonia-1255.png,True,G19887,val,cd426ea6d4cc1223f798d59e793adcac,c02ae32a96cb3dea
Viral Pneumonia-1256.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1256.png,Viral Pneumonia/masks/Viral Pneumonia-1256.png,True,G19888,test,7feb19467a6439bef377c03574f0c004,d17fd658c029386b
Viral Pneumonia-1257.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1257.png,Viral Pneumonia/masks/Viral Pneumonia-1257.png,True,G19889,val,f7ae5a96c2f9ad427451154096a30732,9560069f963b8f3a
Viral Pneumonia-1258.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1258.png,Viral Pneumonia/masks/Viral Pneumonia-1258.png,True,G19890,val,6f950434bedfff69e7fbf7ddad0c7846,916b6e3cf4167065
Viral Pneumonia-1259.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1259.png,Viral Pneumonia/masks/Viral Pneumonia-1259.png,True,G19891,test,5672f0cb841c4ab8eb89c681435febe9,982cc61bc34fd2c7
Viral Pneumonia-126.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-126.png,Viral Pneumonia/masks/Viral Pneumonia-126.png,True,G19892,train,fe6b0b13f447e665a964cab1d29ad1f7,9c7ae30fc11d1c2d
Viral Pneumonia-1260.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1260.png,Viral Pneumonia/masks/Viral Pneumonia-1260.png,True,G19893,train,165bfd0a8038d0886e9ba8230244fdc9,906de73ac61e908f
Viral Pneumonia-1261.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1261.png,Viral Pneumonia/masks/Viral Pneumonia-1261.png,True,G19894,train,e717136c8e02ae187cd5d8d4dcd040a2,946f071b4d3c646e
Viral Pneumonia-1262.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1262.png,Viral Pneumonia/masks/Viral Pneumonia-1262.png,True,G19895,train,44c884d8fe7f04d19e218a40627c2cfd,9029e63aeb693d4c
Viral Pneumonia-1263.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1263.png,Viral Pneumonia/masks/Viral Pneumonia-1263.png,True,G19896,train,d661c947ee1e332047a4e4aebfdd2629,9168e73a8a6b3d64
Viral Pneumonia-1264.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1264.png,Viral Pneumonia/masks/Viral Pneumonia-1264.png,True,G19897,train,de43df051e5dc889d88f8a73d8499502,80e5c61a964eb8df
Viral Pneumonia-1265.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1265.png,Viral Pneumonia/masks/Viral Pneumonia-1265.png,True,G19898,train,31ef5dd9fe98ce157da88efd71e168ff,906be73aab0b3943
Viral Pneumonia-1266.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1266.png,Viral Pneumonia/masks/Viral Pneumonia-1266.png,True,G19899,train,0349ca81fa6132862bc75f16f92dfabb,d12ae75757421e46
Viral Pneumonia-1267.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1267.png,Viral Pneumonia/masks/Viral Pneumonia-1267.png,True,G19900,train,f9ac4c260606f2b997a655abce2336f6,947a6b5a1a5a3d38
Viral Pneumonia-1268.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1268.png,Viral Pneumonia/masks/Viral Pneumonia-1268.png,True,G19901,train,a27c269b18c125f54f3a2838e3a9bf0f,902bef1e98c31e78
Viral Pneumonia-1269.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1269.png,Viral Pneumonia/masks/Viral Pneumonia-1269.png,True,G19902,train,a0d01a894f3e9ebae2bf2e8217b53dc0,d02aaf2b892d0dd7
Viral Pneumonia-127.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-127.png,Viral Pneumonia/masks/Viral Pneumonia-127.png,True,G19903,train,687772530b3a5c7af917d0266f2393d2,916a6f7c51585d07
Viral Pneumonia-1270.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1270.png,Viral Pneumonia/masks/Viral Pneumonia-1270.png,True,G19904,train,0f174bcfe06e276b7001fa9450a42ce5,dd29c35f9e421c16
Viral Pneumonia-1271.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1271.png,Viral Pneumonia/masks/Viral Pneumonia-1271.png,True,G19905,train,f503ecd08eac2c98f0d07c9ca29b1662,9065c71fc34b9743
Viral Pneumonia-1272.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1272.png,Viral Pneumonia/masks/Viral Pneumonia-1272.png,True,G19906,train,c8d726237946565a07263a399cfc979f,9caac35bc30bd04f
Viral Pneumonia-1273.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1273.png,Viral Pneumonia/masks/Viral Pneumonia-1273.png,True,G19907,val,bdc5b0313747b530b4b54882d38c48ac,d4f4967ac30b9983
Viral Pneumonia-1274.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1274.png,Viral Pneumonia/masks/Viral Pneumonia-1274.png,True,G19908,test,c944c65ee92aa6010d89aa07bd34f4d9,80b1c39ade4bbd43
Viral Pneumonia-1275.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1275.png,Viral Pneumonia/masks/Viral Pneumonia-1275.png,True,G19909,test,ff0ee243d4aec07bde4a9f0e3e66c259,d02fef3f4c1c0c45
Viral Pneumonia-1276.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1276.png,Viral Pneumonia/masks/Viral Pneumonia-1276.png,True,G19910,train,9112a7150cc594af144402e96e292f14,917e0f1a0e3b4b3c
Viral Pneumonia-1277.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1277.png,Viral Pneumonia/masks/Viral Pneumonia-1277.png,True,G19911,train,027a786e1c208a1b30e6872892259aab,d434e21bbbc33b90
Viral Pneumonia-1278.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1278.png,Viral Pneumonia/masks/Viral Pneumonia-1278.png,True,G19912,train,74a5d902b9ab6e8a2c92e640253b861e,9125e61eec6b3c43
Viral Pneumonia-1279.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1279.png,Viral Pneumonia/masks/Viral Pneumonia-1279.png,True,G19913,train,91088c96afbccd60b3ec6566d4fe6d18,902be60efe4a3d8c
Viral Pneumonia-128.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-128.png,Viral Pneumonia/masks/Viral Pneumonia-128.png,True,G19914,train,e67a76370458f4fd72035f07a3ac9c55,8127c61ee86bbc4b
Viral Pneumonia-1280.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1280.png,Viral Pneumonia/masks/Viral Pneumonia-1280.png,True,G19915,train,d97b86f6c2b376f47f379d2f0a17547e,d12de35af8693990
Viral Pneumonia-1281.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1281.png,Viral Pneumonia/masks/Viral Pneumonia-1281.png,True,G19916,train,5b5c310623ded0680e4ddcd2b5a7f32e,d1fea61345557885
Viral Pneumonia-1282.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1282.png,Viral Pneumonia/masks/Viral Pneumonia-1282.png,True,G19917,train,7b62dda831134d9b2b4bdd5bdc5d3b25,d0a4c31fe83cca6d
Viral Pneumonia-1283.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1283.png,Viral Pneumonia/masks/Viral Pneumonia-1283.png,True,G19918,train,186b265c88320975f866c46a47fc4f85,812de27ab86d3c78
Viral Pneumonia-1284.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1284.png,Viral Pneumonia/masks/Viral Pneumonia-1284.png,True,G19919,train,ca095bd2d03ac4c75988aea4f06f287b,912be25af85b5949
Viral Pneumonia-1285.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1285.png,Viral Pneumonia/masks/Viral Pneumonia-1285.png,True,G19920,train,7e2e7761dcd2e05423efb7ad5e391614,913fe63e9c4a3c48
Viral Pneumonia-1286.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1286.png,Viral Pneumonia/masks/Viral Pneumonia-1286.png,True,G19921,val,f6afdc0d7353d364ad4ef090c927a724,9121c21bc74ffec2
Viral Pneumonia-1287.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1287.png,Viral Pneumonia/masks/Viral Pneumonia-1287.png,True,G19922,train,6e5bbd6709b42d8d4aca9ffa34f57e52,d4e0a73ba30f830f
Viral Pneumonia-1288.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1288.png,Viral Pneumonia/masks/Viral Pneumonia-1288.png,True,G19923,train,c0574cc9b084b4c9d894893ec0f4a14e,9125e71be52abc62
Viral Pneumonia-1289.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1289.png,Viral Pneumonia/masks/Viral Pneumonia-1289.png,True,G19924,train,688b29c347edd89003a8cfa86a1951f5,d166875a8578b566
Viral Pneumonia-129.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-129.png,Viral Pneumonia/masks/Viral Pneumonia-129.png,True,G19925,test,2dd83787008badaf5ae8bf71f21eaf73,d062c73ae13cf44e
Viral Pneumonia-1290.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1290.png,Viral Pneumonia/masks/Viral Pneumonia-1290.png,True,G19926,test,f99e1e61a3240fb56f3c7b60f7b4fefe,906ab65a8f39353c
Viral Pneumonia-1291.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1291.png,Viral Pneumonia/masks/Viral Pneumonia-1291.png,True,G19927,val,54cd23875b215082665a4709526fe1d3,812fc61ed8693ccd
Viral Pneumonia-1292.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1292.png,Viral Pneumonia/masks/Viral Pneumonia-1292.png,True,G19928,val,38c549943c803d4ab89fadd0cd62797d,90e71711471b8f79
Viral Pneumonia-1293.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1293.png,Viral Pneumonia/masks/Viral Pneumonia-1293.png,True,G19929,test,09b8dd36f050014ab7a317b04a953a07,9137c33bf86938c2
Viral Pneumonia-1294.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1294.png,Viral Pneumonia/masks/Viral Pneumonia-1294.png,True,G19930,train,b26443ce199e3d026728d411dbb1b309,d4fea33bad2b2006
Viral Pneumonia-1295.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1295.png,Viral Pneumonia/masks/Viral Pneumonia-1295.png,True,G19931,train,d3df50e7db513bdc6da6991a102b322d,91e1c73e964ab18e
Viral Pneumonia-1296.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1296.png,Viral Pneumonia/masks/Viral Pneumonia-1296.png,True,G19932,train,82796369750a3f8eba52bd3ca5d553ed,913ac61ab8cf78b1
Viral Pneumonia-1297.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1297.png,Viral Pneumonia/masks/Viral Pneumonia-1297.png,True,G19933,train,2615d038638708f9ccc8f07ef011de9b,914ec65ae578b14e
Viral Pneumonia-1298.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1298.png,Viral Pneumonia/masks/Viral Pneumonia-1298.png,True,G19934,train,cf1a2ccc8fdef96462bb48b448a19dd6,9125c71eca7b3cc2
Viral Pneumonia-1299.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1299.png,Viral Pneumonia/masks/Viral Pneumonia-1299.png,True,G19935,train,97aaa40d3902dafae81394209a467cad,c0fce70f9aa78a09
Viral Pneumonia-13.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-13.png,Viral Pneumonia/masks/Viral Pneumonia-13.png,True,G19936,train,0a6cf73b05cddfb4cf718e5ef27accf3,d02bea4f1e430f3c
Viral Pneumonia-130.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-130.png,Viral Pneumonia/masks/Viral Pneumonia-130.png,True,G19937,train,edde31a22450f56d60df820b1d6737c4,9420c31fee1eb60f
Viral Pneumonia-1300.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1300.png,Viral Pneumonia/masks/Viral Pneumonia-1300.png,True,G19938,train,f74bec877725b1f33e6efcb279b1be33,d02de23db8c82d9d
Viral Pneumonia-1301.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1301.png,Viral Pneumonia/masks/Viral Pneumonia-1301.png,True,G19939,train,07b811f5852422b1e661f04525c5ecae,d127e63abc683c68
Viral Pneumonia-1302.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1302.png,Viral Pneumonia/masks/Viral Pneumonia-1302.png,True,G19940,train,9bd46877fd9e7a571bd47c635e21a99c,d03ac30bbdcd3cb0
Viral Pneumonia-1303.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1303.png,Viral Pneumonia/masks/Viral Pneumonia-1303.png,True,G19941,train,a26ff61c6e3c29285d569136c060932d,906de61eec2d29c5
Viral Pneumonia-1304.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1304.png,Viral Pneumonia/masks/Viral Pneumonia-1304.png,True,G19942,val,41159b3babbe4436e5984133f5249313,9561e73ac23a954e
Viral Pneumonia-1305.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1305.png,Viral Pneumonia/masks/Viral Pneumonia-1305.png,True,G19943,train,d61a6aa9bc16cd532195fd2d5e808201,9c3a931fd51c3c1a
Viral Pneumonia-1306.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1306.png,Viral Pneumonia/masks/Viral Pneumonia-1306.png,True,G19944,train,8b41f43b7cda488dee15cb162fd0c7ff,9478e53fba830b38
Viral Pneumonia-1307.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1307.png,Viral Pneumonia/masks/Viral Pneumonia-1307.png,True,G19945,val,43c2172726d28cf2feb0650005a4b686,912fe71bf04b3cc0
Viral Pneumonia-1308.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1308.png,Viral Pneumonia/masks/Viral Pneumonia-1308.png,True,G19946,val,ad3f63c5d05a72ff3ab63da2bf9a6863,912fe63aba4b3a60
Viral Pneumonia-1309.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1309.png,Viral Pneumonia/masks/Viral Pneumonia-1309.png,True,G19947,test,ebc117f0ca583910561bcd49d838198f,d0e2c25bc758ed0e
Viral Pneumonia-131.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-131.png,Viral Pneumonia/masks/Viral Pneumonia-131.png,True,G19948,test,e77b3397ab90b9326385f921dbcd7b62,d02de3a99acb9e48
Viral Pneumonia-1310.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1310.png,Viral Pneumonia/masks/Viral Pneumonia-1310.png,True,G19949,train,493c8bd7a9a9c47e58ce806456210cc6,9511e31ef64a3ce8
Viral Pneumonia-1311.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1311.png,Viral Pneumonia/masks/Viral Pneumonia-1311.png,True,G19950,train,75b2a9b97020abf70448cee22b51e5ae,913bc65a584b3d33
Viral Pneumonia-1312.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1312.png,Viral Pneumonia/masks/Viral Pneumonia-1312.png,True,G19951,test,0423b6403fa47bacb7a79b8426294859,912de63af86b3849
Viral Pneumonia-1313.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1313.png,Viral Pneumonia/masks/Viral Pneumonia-1313.png,True,G19952,train,16f250c3bedaca0d10a509734653c31c,d035c31f9c4b3c6a
Viral Pneumonia-1314.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1314.png,Viral Pneumonia/masks/Viral Pneumonia-1314.png,True,G19953,train,7069644ac7cd2ed64bda6e3dc8e49339,9169c73ec4ce38d2
Viral Pneumonia-1315.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1315.png,Viral Pneumonia/masks/Viral Pneumonia-1315.png,True,G19954,train,e9acab1ca548b751da0c0dc69d93f2d0,c02fe71fec6d2c02
Viral Pneumonia-1316.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1316.png,Viral Pneumonia/masks/Viral Pneumonia-1316.png,True,G19955,train,b2e577b99496a513916dacecc2ef209a,d5ffaf0203155705
Viral Pneumonia-1317.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1317.png,Viral Pneumonia/masks/Viral Pneumonia-1317.png,True,G19956,train,686f056e115d764ac8a111657d527411,d072c75b952da50e
Viral Pneumonia-1318.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1318.png,Viral Pneumonia/masks/Viral Pneumonia-1318.png,True,G19957,train,f5a52f1259c55672b2423454422ffb9f,8137e61ab8ca3dd8
Viral Pneumonia-1319.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1319.png,Viral Pneumonia/masks/Viral Pneumonia-1319.png,True,G19958,train,5945d294e68bfcc950d2ff710678724e,d025c63fde4b3c60
Viral Pneumonia-132.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-132.png,Viral Pneumonia/masks/Viral Pneumonia-132.png,True,G19959,train,64c2e44bbacc29731cd3ac727de989b7,9127e73ab04bb44e
Viral Pneumonia-1320.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1320.png,Viral Pneumonia/masks/Viral Pneumonia-1320.png,True,G19960,train,bd25d0e1938c5d8272173df237e7c44c,95e1c71ee82db04e
Viral Pneumonia-1321.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1321.png,Viral Pneumonia/masks/Viral Pneumonia-1321.png,True,G19961,train,4abd6f52e75834ac830311cdb42d165b,90a6c33bc63cc66d
Viral Pneumonia-1322.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1322.png,Viral Pneumonia/masks/Viral Pneumonia-1322.png,True,G19962,test,e4418c49660c7eba83cb74eaa430d509,d42be61edf491d40
Viral Pneumonia-1323.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1323.png,Viral Pneumonia/masks/Viral Pneumonia-1323.png,True,G19963,train,21e411981177dbae3a1013c6e405d87c,907b471e5e4b1d49
Viral Pneumonia-1324.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1324.png,Viral Pneumonia/masks/Viral Pneumonia-1324.png,True,G19964,train,145d02dac14c51b420c125688a8ff8a2,d066e73ff4291c03
Viral Pneumonia-1325.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1325.png,Viral Pneumonia/masks/Viral Pneumonia-1325.png,True,G19965,train,9a20a71a66f7524e5f0cbe595eb537f0,9565c63ace5a984e
Viral Pneumonia-1326.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1326.png,Viral Pneumonia/masks/Viral Pneumonia-1326.png,True,G19966,val,1fa07c1d787f351c1a4f74b922842bad,9125c71eca4ebc4e
Viral Pneumonia-1327.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1327.png,Viral Pneumonia/masks/Viral Pneumonia-1327.png,True,G19967,val,7204799b214b1ac29b1709311fb202d2,d47e835bab3cc034
Viral Pneumonia-1328.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1328.png,Viral Pneumonia/masks/Viral Pneumonia-1328.png,True,G19968,val,259ceb561b9bdb8c651df29ca0a94066,d4f5a71ae53ba408
Viral Pneumonia-1329.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1329.png,Viral Pneumonia/masks/Viral Pneumonia-1329.png,True,G19969,train,503500ccdea513a35179c4d9da88632a,d02fc31fe8299d49
Viral Pneumonia-133.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-133.png,Viral Pneumonia/masks/Viral Pneumonia-133.png,True,G19970,train,fe14b32baab3a6bf076a027a10c7a829,d464c20bc71bb8cf
Viral Pneumonia-1330.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1330.png,Viral Pneumonia/masks/Viral Pneumonia-1330.png,True,G19971,test,7ad59cb4c3ad6cf74715ee137dc9b2be,94eaab3bc51ce02e
Viral Pneumonia-1331.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1331.png,Viral Pneumonia/masks/Viral Pneumonia-1331.png,True,G19972,test,84da65a0fcd7efa24d1d2860e28fa5fa,94eea7028f1b8b1d
Viral Pneumonia-1332.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1332.png,Viral Pneumonia/masks/Viral Pneumonia-1332.png,True,G19973,train,edfcb5d8876012d381382f0f6c3593d2,917bc71e0b5b1945
Viral Pneumonia-1333.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1333.png,Viral Pneumonia/masks/Viral Pneumonia-1333.png,True,G19974,train,e5929a4623ddfedbd33954b644e537e5,80e1875acf3cc33d
Viral Pneumonia-1334.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1334.png,Viral Pneumonia/masks/Viral Pneumonia-1334.png,True,G19975,test,d047637bea0e8ad68a6bd77c0f264c29,d064e61ae50fb14f
Viral Pneumonia-1335.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1335.png,Viral Pneumonia/masks/Viral Pneumonia-1335.png,True,G19976,train,a1aae055967905835b06973d94cc8aa6,95a7c698e938c339
Viral Pneumonia-1336.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1336.png,Viral Pneumonia/masks/Viral Pneumonia-1336.png,True,G19977,train,d44475d255e9f87f67a27af8cf66f89f,d462a75ac31fc117
Viral Pneumonia-1337.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1337.png,Viral Pneumonia/masks/Viral Pneumonia-1337.png,True,G19978,train,ea60e4b5db2131408446d4634a7e6fe1,946ce33ae56d3c05
Viral Pneumonia-1338.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1338.png,Viral Pneumonia/masks/Viral Pneumonia-1338.png,True,G19979,train,d5a650aed16f15a5c9243869a66de6bf,d42fe30bef680c85
Viral Pneumonia-1339.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1339.png,Viral Pneumonia/masks/Viral Pneumonia-1339.png,True,G19980,train,ca1a679488a137a73df7e507f7c97b1c,d46ac71eeb2d2c44
Viral Pneumonia-134.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-134.png,Viral Pneumonia/masks/Viral Pneumonia-134.png,True,G19981,train,72d13186151e398877c34cf696216bdf,c021e71afe683d69
Viral Pneumonia-1340.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1340.png,Viral Pneumonia/masks/Viral Pneumonia-1340.png,True,G19982,train,8461941c980ec5c451d66494caebf67d,d06bcb1bc11bc743
Viral Pneumonia-1341.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1341.png,Viral Pneumonia/masks/Viral Pneumonia-1341.png,True,G19983,train,1807f33d587f4e356fe31c755742c39d,d464931b853de996
Viral Pneumonia-1342.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1342.png,Viral Pneumonia/masks/Viral Pneumonia-1342.png,True,G19984,train,d0f5c885438e702d67c14c141b77592b,d4b4c21bc64bb4cb
Viral Pneumonia-1343.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1343.png,Viral Pneumonia/masks/Viral Pneumonia-1343.png,True,G19985,test,ed25fc9d62436cb4cf4205feda5bc053,c062c31bc34fb5cb
Viral Pneumonia-1344.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1344.png,Viral Pneumonia/masks/Viral Pneumonia-1344.png,True,G19986,train,90f2fd9447ed2796be40a973b935dd32,d02de71e96cb1dc0
Viral Pneumonia-1345.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-1345.png,Viral Pneumonia/masks/Viral Pneumonia-1345.png,True,G19987,train,dccc00ca5707610be8a300d71f7bbb03,d42de31f9fc92c80
Viral Pneumonia-135.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-135.png,Viral Pneumonia/masks/Viral Pneumonia-135.png,True,G19988,train,43a86ac20acdf3d265d6be3555c25d3b,9029e2bee26ee04f
Viral Pneumonia-136.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-136.png,Viral Pneumonia/masks/Viral Pneumonia-136.png,True,G19989,val,649c0b083c73916e1198e6982e0bc706,902bc31ed65fd443
Viral Pneumonia-137.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-137.png,Viral Pneumonia/masks/Viral Pneumonia-137.png,True,G19990,val,67a082790ebd7a82869858bc44b3fe05,d13bd61a19d3139c
Viral Pneumonia-138.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-138.png,Viral Pneumonia/masks/Viral Pneumonia-138.png,True,G19991,val,4e8f08c38ca51be5bd40a928763aab24,d474c61a934bb54e
Viral Pneumonia-139.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-139.png,Viral Pneumonia/masks/Viral Pneumonia-139.png,True,G19992,test,5894657ff5e6b1126f3cdd8ceadd4419,c012e79b9bcd3964
Viral Pneumonia-14.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-14.png,Viral Pneumonia/masks/Viral Pneumonia-14.png,True,G19993,train,da5d9e3227be8f946bcf05caaf03df7e,956ac63c533d34a5
Viral Pneumonia-140.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-140.png,Viral Pneumonia/masks/Viral Pneumonia-140.png,True,G19994,train,82f999a0bdb0a6003351d5624b7a00bd,d53ce0292f92638f
Viral Pneumonia-141.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-141.png,Viral Pneumonia/masks/Viral Pneumonia-141.png,True,G19995,test,dba4683ddd1c45d9d8e3ad3fb6e0ea30,8129e64e964a3f76
Viral Pneumonia-142.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-142.png,Viral Pneumonia/masks/Viral Pneumonia-142.png,True,G19996,train,49470f1a7ce182280dfd884843e760dc,986de73ac6389278
Viral Pneumonia-143.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-143.png,Viral Pneumonia/masks/Viral Pneumonia-143.png,True,G19997,train,0a9d258255ae703f048fccbb379698f0,80bbd61e965695d4
Viral Pneumonia-144.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-144.png,Viral Pneumonia/masks/Viral Pneumonia-144.png,True,G19998,train,c810d9eadbdb7db5728a324654f1a14a,906fef3cc238c13c
Viral Pneumonia-145.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-145.png,Viral Pneumonia/masks/Viral Pneumonia-145.png,True,G19999,train,15d4fc6d668568249465980e6d71c14f,957b6a3e5e661848
Viral Pneumonia-146.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-146.png,Viral Pneumonia/masks/Viral Pneumonia-146.png,True,G20000,train,5ba8152e6622b52b1208a390f84bd78c,946b6a3ed652585a
Viral Pneumonia-147.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-147.png,Viral Pneumonia/masks/Viral Pneumonia-147.png,True,G20001,train,ff35265a046bde1c3d2d45e6612238a0,947a6d1c4a286c7f
Viral Pneumonia-148.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-148.png,Viral Pneumonia/masks/Viral Pneumonia-148.png,True,G20002,train,d2c8784d629a1c535bc2723d67f96431,d42de21bc30bf067
Viral Pneumonia-149.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-149.png,Viral Pneumonia/masks/Viral Pneumonia-149.png,True,G20003,train,7380d233317103e1024b5f6c8268f013,812bd65ed6681d69
Viral Pneumonia-15.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-15.png,Viral Pneumonia/masks/Viral Pneumonia-15.png,True,G20004,train,d1d1eb65e6f6e572c977a5e03ecaeab1,917bc63ed2c23994
Viral Pneumonia-150.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-150.png,Viral Pneumonia/masks/Viral Pneumonia-150.png,True,G20005,train,093699278a28e5d5012970f492529b95,9821e33ff44f3461
Viral Pneumonia-151.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-151.png,Viral Pneumonia/masks/Viral Pneumonia-151.png,True,G20006,val,b08da33b7c7d4a4fcc403b62c0054f25,9838e14a3de44fb5
Viral Pneumonia-152.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-152.png,Viral Pneumonia/masks/Viral Pneumonia-152.png,True,G20007,val,0106acd24ef33c445afae12a06aa359c,80e5c2bac339c67e
Viral Pneumonia-153.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-153.png,Viral Pneumonia/masks/Viral Pneumonia-153.png,True,G20008,test,f044201461945c402901cb4e5750e3eb,d428e20baf6d4ce5
Viral Pneumonia-154.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-154.png,Viral Pneumonia/masks/Viral Pneumonia-154.png,True,G20009,test,64f72715579883da978ea36f0f41b31a,9165c61eea3b38e1
Viral Pneumonia-155.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-155.png,Viral Pneumonia/masks/Viral Pneumonia-155.png,True,G20010,val,f5a44e2f218beacf5ab79984251cd2cf,d4fe830109a5adaf
Viral Pneumonia-156.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-156.png,Viral Pneumonia/masks/Viral Pneumonia-156.png,True,G20011,train,d78ccc38bb5b834b7b0e3687c507d26a,81becf1ec9996984
Viral Pneumonia-157.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-157.png,Viral Pneumonia/masks/Viral Pneumonia-157.png,True,G20012,train,35ba352314dc1c122154d84e01a2fae9,9d6a87384d9538ad
Viral Pneumonia-158.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-158.png,Viral Pneumonia/masks/Viral Pneumonia-158.png,True,G20013,train,559ff07016ed1f984a95917b462110c1,803bb5cbc3a17d29
Viral Pneumonia-159.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-159.png,Viral Pneumonia/masks/Viral Pneumonia-159.png,True,G20014,train,c3fe5b2ba56e8f321cd8e4d40ecb0a01,d86ec31bc33d6885
Viral Pneumonia-16.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-16.png,Viral Pneumonia/masks/Viral Pneumonia-16.png,True,G20015,train,41d217aaa79718b74b241aeef2ab2e69,d47c873b831b830f
Viral Pneumonia-160.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-160.png,Viral Pneumonia/masks/Viral Pneumonia-160.png,True,G20016,train,e3a2c842c22e2ed7e587d4fba7926d97,902bf75ee44878d1
Viral Pneumonia-161.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-161.png,Viral Pneumonia/masks/Viral Pneumonia-161.png,True,G20017,train,f5f3759897bc653b3440202756e7132d,d06be71ee9491d41
Viral Pneumonia-162.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-162.png,Viral Pneumonia/masks/Viral Pneumonia-162.png,True,G20018,train,d1d424761f829cfda3e8cf9aa41a60fc,d46fe31a834b4f41
Viral Pneumonia-163.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-163.png,Viral Pneumonia/masks/Viral Pneumonia-163.png,True,G20019,train,26afab207af4cb20ae68e6bfbadea498,986fe71ac5491d43
Viral Pneumonia-164.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-164.png,Viral Pneumonia/masks/Viral Pneumonia-164.png,True,G20020,train,4d33c8fe11f5903a58e612a048f0f8e7,912ce22fe44bbcc3
Viral Pneumonia-165.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-165.png,Viral Pneumonia/masks/Viral Pneumonia-165.png,True,G20021,train,0fe50b3832a2218bbdc6d8bc04e147d9,d3754449199b2bb3
Viral Pneumonia-166.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-166.png,Viral Pneumonia/masks/Viral Pneumonia-166.png,True,G20022,train,167809497c57d8eb397568afd41ec7db,916fef5ec64a0541
Viral Pneumonia-167.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-167.png,Viral Pneumonia/masks/Viral Pneumonia-167.png,True,G20023,train,2759c9f9014205e200be6535a97c0bf2,906ac75fc5c9c8c5
Viral Pneumonia-168.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-168.png,Viral Pneumonia/masks/Viral Pneumonia-168.png,True,G20024,test,9567dff091a03520a4159079ae06d99a,d02f251d55686ce7
Viral Pneumonia-169.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-169.png,Viral Pneumonia/masks/Viral Pneumonia-169.png,True,G20025,test,9e0ba2c45f32e329003fbed2485f4ff3,d4f6c71b8d1b0d0c
Viral Pneumonia-17.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-17.png,Viral Pneumonia/masks/Viral Pneumonia-17.png,True,G20026,test,7b3cbb86de25cd0c0ad8ecfa9f082a09,d12de61ed4423e6a
Viral Pneumonia-170.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-170.png,Viral Pneumonia/masks/Viral Pneumonia-170.png,True,G20027,train,0fc0c1679f38c683055aab0825811985,9029e5673eb4c2a7
Viral Pneumonia-171.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-171.png,Viral Pneumonia/masks/Viral Pneumonia-171.png,True,G20028,train,9807900bee0a64bbdb4b48780d129811,9129eb3b795919d0
Viral Pneumonia-172.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-172.png,Viral Pneumonia/masks/Viral Pneumonia-172.png,True,G20029,train,56d169a71d4eef747285c782dfeade8f,946b4f1e480f7c4d
Viral Pneumonia-173.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-173.png,Viral Pneumonia/masks/Viral Pneumonia-173.png,True,G20030,val,9d1eb835d1d07b716f29e3e79ebf297c,9469e31ed05a3ac7
Viral Pneumonia-174.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-174.png,Viral Pneumonia/masks/Viral Pneumonia-174.png,True,G20031,val,237ac8ac9c829453a454c6a441bf70ae,9174c21bdb1fa44b
Viral Pneumonia-175.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-175.png,Viral Pneumonia/masks/Viral Pneumonia-175.png,True,G20032,val,594f283f2f0956c9a769cf06e36b12d5,c03d131b1dbcb8dc
Viral Pneumonia-176.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-176.png,Viral Pneumonia/masks/Viral Pneumonia-176.png,True,G20033,train,b27a0894900ff4d3d3abbd5ff91893d2,9129c60efae32f38
Viral Pneumonia-177.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-177.png,Viral Pneumonia/masks/Viral Pneumonia-177.png,True,G20034,train,f4d6476a7f4abcce0a88cdce0dc1715b,c46ca73baf2d2c0c
Viral Pneumonia-178.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-178.png,Viral Pneumonia/masks/Viral Pneumonia-178.png,True,G20035,train,f441438e3930a8f5d43631078d02d60e,817da73e8f0b0d0d
Viral Pneumonia-179.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-179.png,Viral Pneumonia/masks/Viral Pneumonia-179.png,True,G20036,train,e97c9891753bf76543c73b20203c1772,807ce51abacf3aa4
Viral Pneumonia-18.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-18.png,Viral Pneumonia/masks/Viral Pneumonia-18.png,True,G20037,train,1c5b33895cfb3d5481ce7a7cf95fae4e,993de71ec44336c2
Viral Pneumonia-180.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-180.png,Viral Pneumonia/masks/Viral Pneumonia-180.png,True,G20038,train,75da8c2be9a6d585093a79a362d0d11b,d054e71ae14f98e5
Viral Pneumonia-181.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-181.png,Viral Pneumonia/masks/Viral Pneumonia-181.png,True,G20039,train,ecd41cd1a8d06f4896bb29070b308f47,d51ce10a79e58a79
Viral Pneumonia-182.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-182.png,Viral Pneumonia/masks/Viral Pneumonia-182.png,True,G20040,train,5ca8b3006ec8476dcae0a643ff0c5454,d052c35be57db043
Viral Pneumonia-183.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-183.png,Viral Pneumonia/masks/Viral Pneumonia-183.png,True,G20041,train,73bf52155f5c01d16a012e9eb8f2a162,9565c63cebbc2885
Viral Pneumonia-184.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-184.png,Viral Pneumonia/masks/Viral Pneumonia-184.png,True,G20042,train,bbc28322576a9126a3ce6ef1f97b0a09,d5f45a4ae15aa51a
Viral Pneumonia-185.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-185.png,Viral Pneumonia/masks/Viral Pneumonia-185.png,True,G20043,train,bb8d4cf604f3e537d9d394d21d30fdab,d42b6b5f1e402e1e
Viral Pneumonia-186.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-186.png,Viral Pneumonia/masks/Viral Pneumonia-186.png,True,G20044,train,824e9be1b2d39826643e4bc89e6a1c27,907f5716c35b5443
Viral Pneumonia-187.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-187.png,Viral Pneumonia/masks/Viral Pneumonia-187.png,True,G20045,test,c16020833227cd927d28a113fd4ea842,d074e71ba56d28c6
Viral Pneumonia-188.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-188.png,Viral Pneumonia/masks/Viral Pneumonia-188.png,True,G20046,val,f6566061c59fb32e9aa1d0b6ef61c692,9129e62af86a396d
Viral Pneumonia-189.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-189.png,Viral Pneumonia/masks/Viral Pneumonia-189.png,True,G20047,test,515c1020190e735e14403d132270fb8b,d03faf1f892d28c2
Viral Pneumonia-19.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-19.png,Viral Pneumonia/masks/Viral Pneumonia-19.png,True,G20048,val,ead647bd97f54084e1936936ffc32c5e,d4ef07198719817b
Viral Pneumonia-190.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-190.png,Viral Pneumonia/masks/Viral Pneumonia-190.png,True,G20049,val,0adc56ab3a690bb8ee98b7a633eb11af,917bc61a524e5c4f
Viral Pneumonia-191.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-191.png,Viral Pneumonia/masks/Viral Pneumonia-191.png,True,G20050,test,6cfb6386609423631e3fe1bda90a7fe0,9008e32fbde0afb8
Viral Pneumonia-192.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-192.png,Viral Pneumonia/masks/Viral Pneumonia-192.png,True,G20051,train,278e6539e3617e83f71f8435af585c0d,913fde5ed4403469
Viral Pneumonia-193.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-193.png,Viral Pneumonia/masks/Viral Pneumonia-193.png,True,G20052,train,0700c377a3a670ea4aad4d40d23992aa,d0bdc23dec3c382c
Viral Pneumonia-194.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-194.png,Viral Pneumonia/masks/Viral Pneumonia-194.png,True,G20053,train,86084f5ad434df3f0770407f8898f478,856ae27ae05e7855
Viral Pneumonia-195.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-195.png,Viral Pneumonia/masks/Viral Pneumonia-195.png,True,G20054,train,38727703bedb23016990d7993337a8d8,d0acafaf2be8282c
Viral Pneumonia-196.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-196.png,Viral Pneumonia/masks/Viral Pneumonia-196.png,True,G20055,train,281a34e341d81d1ca958f490d955ec8e,88b1c33aa55eb547
Viral Pneumonia-197.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-197.png,Viral Pneumonia/masks/Viral Pneumonia-197.png,True,G20056,train,efe0b1b86dad8f0a2d9a01da3b1be830,917e8f1e1d4d1c4c
Viral Pneumonia-198.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-198.png,Viral Pneumonia/masks/Viral Pneumonia-198.png,True,G20057,train,4377dd7c393f63396959b5d1b7851469,d42de23fe90d2c49
Viral Pneumonia-199.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-199.png,Viral Pneumonia/masks/Viral Pneumonia-199.png,True,G20058,train,de03c1c4efd89058bbcae7c1245ef75e,91e8a73ae93c6a2c
Viral Pneumonia-2.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-2.png,Viral Pneumonia/masks/Viral Pneumonia-2.png,True,G20059,train,5626ee812c94c7bf3e16753c9bd887ce,d07ca73dad2f2882
Viral Pneumonia-20.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-20.png,Viral Pneumonia/masks/Viral Pneumonia-20.png,True,G20060,train,35601e7fdd0a6cf09a3eb0e0e4d9b90d,916f4618c75ad65a
Viral Pneumonia-200.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-200.png,Viral Pneumonia/masks/Viral Pneumonia-200.png,True,G20061,train,0a0f0e66756eadfae05b726673f090bf,d420c30bef4bbc4b
Viral Pneumonia-201.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-201.png,Viral Pneumonia/masks/Viral Pneumonia-201.png,True,G20062,train,121094bb232e08257049100befca3d5e,80816fe83e1cefc9
Viral Pneumonia-202.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-202.png,Viral Pneumonia/masks/Viral Pneumonia-202.png,True,G20063,train,a11831e98ede9112a1eb83d79361908c,9410634f784b5b6f
Viral Pneumonia-203.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-203.png,Viral Pneumonia/masks/Viral Pneumonia-203.png,True,G20064,train,36f486436dbe0977046587ff9f4989a3,90c16fa7ed31e838
Viral Pneumonia-204.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-204.png,Viral Pneumonia/masks/Viral Pneumonia-204.png,True,G20065,test,e9463c6e4fcd773cbaa6c92599f308de,803de02bbeeaa51e
Viral Pneumonia-205.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-205.png,Viral Pneumonia/masks/Viral Pneumonia-205.png,True,G20066,val,1c0061cef331d551648b9f677c2f3bbe,81a5e31ff08f86a9
Viral Pneumonia-206.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-206.png,Viral Pneumonia/masks/Viral Pneumonia-206.png,True,G20067,test,f40973413ccfb0ba07a3fd419a4af079,d42f0b1fc91dc4c6
Viral Pneumonia-207.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-207.png,Viral Pneumonia/masks/Viral Pneumonia-207.png,True,G20068,test,fddc8e440d0f6389c6ea48ebbd0c3bc0,d46e851b9f1f09a4
Viral Pneumonia-208.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-208.png,Viral Pneumonia/masks/Viral Pneumonia-208.png,True,G20069,train,db753afce2a1a99336a598b6ffd05823,901ce1e33bb8e1c7
Viral Pneumonia-209.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-209.png,Viral Pneumonia/masks/Viral Pneumonia-209.png,True,G20070,val,00d453cc731217bc914b232e2f1cdc3e,946b4b1f490f781e
Viral Pneumonia-21.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-21.png,Viral Pneumonia/masks/Viral Pneumonia-21.png,True,G20071,train,8db63b33f43bd2001d85d418cb12bda3,9065c71ee839cc3d
Viral Pneumonia-210.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-210.png,Viral Pneumonia/masks/Viral Pneumonia-210.png,True,G20072,train,3729a46cb6d29c72eed5198202ffb7fc,d4fdf5198549614c
Viral Pneumonia-211.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-211.png,Viral Pneumonia/masks/Viral Pneumonia-211.png,True,G20073,val,4577877513eafe5ca0b168fc0e811c0b,d068e41b954b3d57
Viral Pneumonia-212.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-212.png,Viral Pneumonia/masks/Viral Pneumonia-212.png,True,G20074,train,4b6739bdf36054da4d5e0e4095bc4e3a,d4eae75a514b7842
Viral Pneumonia-213.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-213.png,Viral Pneumonia/masks/Viral Pneumonia-213.png,True,G20075,train,92c1049b35b12bae8ea5835bc52b62b7,d460c30fe10f78f6
Viral Pneumonia-214.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-214.png,Viral Pneumonia/masks/Viral Pneumonia-214.png,True,G20076,train,084105723943df132da3beb9e69822ca,912be33ae8693d49
Viral Pneumonia-215.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-215.png,Viral Pneumonia/masks/Viral Pneumonia-215.png,True,G20077,train,2fafe63484791fc2f8e63aa37a8410c2,9070871acf9f6996
Viral Pneumonia-216.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-216.png,Viral Pneumonia/masks/Viral Pneumonia-216.png,True,G20078,train,1a3619156dcd36c3186dcd0d44997161,9025e21eea79bd49
Viral Pneumonia-217.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-217.png,Viral Pneumonia/masks/Viral Pneumonia-217.png,True,G20079,train,de50b569f5447931e9beddda926f33c7,9065c71ae71b867a
Viral Pneumonia-218.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-218.png,Viral Pneumonia/masks/Viral Pneumonia-218.png,True,G20080,train,5c71696d7fda348bd352ae22b8145da5,9135c21fd64b9e4a
Viral Pneumonia-219.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-219.png,Viral Pneumonia/masks/Viral Pneumonia-219.png,True,G20081,train,0b24c2681ce5449a342f44577f1c5648,c027c31bed6d2cc6
Viral Pneumonia-22.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-22.png,Viral Pneumonia/masks/Viral Pneumonia-22.png,True,G20082,train,038ff29e52c8124c8189f966963ddf4d,c137e21b9e69b0ca
Viral Pneumonia-220.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-220.png,Viral Pneumonia/masks/Viral Pneumonia-220.png,True,G20083,train,bae595880046d28d63783198a572ff76,9139e1433f546b87
Viral Pneumonia-221.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-221.png,Viral Pneumonia/masks/Viral Pneumonia-221.png,True,G20084,train,47f31ed0b0303dfe19db3633e0677411,d46be39ed25a4845
Viral Pneumonia-222.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-222.png,Viral Pneumonia/masks/Viral Pneumonia-222.png,True,G20085,train,75d691986ba416fc6a3be258b019a02a,802af1433df0c3ef
Viral Pneumonia-223.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-223.png,Viral Pneumonia/masks/Viral Pneumonia-223.png,True,G20086,train,500358be59e129ae8411775582feb59e,d02dc61fae6b3c41
Viral Pneumonia-224.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-224.png,Viral Pneumonia/masks/Viral Pneumonia-224.png,True,G20087,val,4251b018e5b073ee5b9445e2f1ac580c,c0bfab0b0d0d2d3e
Viral Pneumonia-225.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-225.png,Viral Pneumonia/masks/Viral Pneumonia-225.png,True,G20088,val,5b92c4818a1c942d0870fe657c94fbba,d42de32bec682c69
Viral Pneumonia-226.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-226.png,Viral Pneumonia/masks/Viral Pneumonia-226.png,True,G20089,val,d989e1823aea6723e46b7f69e875fcbf,d028e14b3ce54ebe
Viral Pneumonia-227.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-227.png,Viral Pneumonia/masks/Viral Pneumonia-227.png,True,G20090,test,fa5fcb678ecfac95bbfe7290aed00ae6,d4f4a238874af32e
Viral Pneumonia-228.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-228.png,Viral Pneumonia/masks/Viral Pneumonia-228.png,True,G20091,test,47c5687b64eefc001d7da6daf70c06e5,d456be4a8770a3b1
Viral Pneumonia-229.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-229.png,Viral Pneumonia/masks/Viral Pneumonia-229.png,True,G20092,test,f1f5fe28a7d73b7a1b97b6f1e36d8202,d0f4c71bc539944b
Viral Pneumonia-23.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-23.png,Viral Pneumonia/masks/Viral Pneumonia-23.png,True,G20093,train,e0a83bfabf2821eff70df4526c15f20e,907f4f1e185a5b58
Viral Pneumonia-230.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-230.png,Viral Pneumonia/masks/Viral Pneumonia-230.png,True,G20094,train,27c2cdcf3c58d26448eb82b512086e0d,c074c24bd76b914f
Viral Pneumonia-231.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-231.png,Viral Pneumonia/masks/Viral Pneumonia-231.png,True,G20095,val,6b39af4006a444926a01b2438b4eec8e,8165c63ae23bb34b
Viral Pneumonia-232.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-232.png,Viral Pneumonia/masks/Viral Pneumonia-232.png,True,G20096,train,4853571c3412ae50372ffef2c560c9b5,c029e21fd84f3d78
Viral Pneumonia-233.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-233.png,Viral Pneumonia/masks/Viral Pneumonia-233.png,True,G20097,train,3748b3f3d071ca24a77d9b5ab403c319,d0af7d1d480b2a4f
Viral Pneumonia-234.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-234.png,Viral Pneumonia/masks/Viral Pneumonia-234.png,True,G20098,train,9b33b516c6d95101e8fe7a2fc63ad387,802bef3ef44eb083
Viral Pneumonia-235.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-235.png,Viral Pneumonia/masks/Viral Pneumonia-235.png,True,G20099,train,be73a8defd86d8df3e9141d4319691ed,d02bf61e5c493959
Viral Pneumonia-236.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-236.png,Viral Pneumonia/masks/Viral Pneumonia-236.png,True,G20100,train,bde5e618f2ffcdb9ed3b41e255a70ef7,d06be71ede433842
Viral Pneumonia-237.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-237.png,Viral Pneumonia/masks/Viral Pneumonia-237.png,True,G20101,test,211d1c7acd1606863b49be71cf6ab709,9168463e4f1b79c6
Viral Pneumonia-238.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-238.png,Viral Pneumonia/masks/Viral Pneumonia-238.png,True,G20102,train,fa5b27f387b84464f64ff44607c4c0b3,d03afb165e143a1e
Viral Pneumonia-239.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-239.png,Viral Pneumonia/masks/Viral Pneumonia-239.png,True,G20103,train,50ba35d47faa6b5c1f67c3799cc7d2f6,d01b56525e5a5b3a
Viral Pneumonia-24.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-24.png,Viral Pneumonia/masks/Viral Pneumonia-24.png,True,G20104,train,66b34bee9e62846a36571586e81e08a1,812beedad4105d73
Viral Pneumonia-240.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-240.png,Viral Pneumonia/masks/Viral Pneumonia-240.png,True,G20105,train,28e3c50c4dc99cc1a1c6fa0e2d093d71,957eef3c02067847
Viral Pneumonia-241.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-241.png,Viral Pneumonia/masks/Viral Pneumonia-241.png,True,G20106,train,f82460471d8f6cc3bc2538cf203c71f6,9074c71a964bb9c7
Viral Pneumonia-242.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-242.png,Viral Pneumonia/masks/Viral Pneumonia-242.png,True,G20107,val,3e9308ed064657eb158820e626897e61,d121e60bbecab21e
Viral Pneumonia-243.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-243.png,Viral Pneumonia/masks/Viral Pneumonia-243.png,True,G20108,val,5ed92f712c9bda8762e2aa18e3f64f61,c4f4930be56db44c
Viral Pneumonia-244.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-244.png,Viral Pneumonia/masks/Viral Pneumonia-244.png,True,G20109,val,254a1afef84d3c1d48700d13cc4de1f0,912fe31eec693c41
Viral Pneumonia-245.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-245.png,Viral Pneumonia/masks/Viral Pneumonia-245.png,True,G20110,test,c2d03b3e2cbb7e7e0b68a42dbd461fec,d424e32fed2da849
Viral Pneumonia-246.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-246.png,Viral Pneumonia/masks/Viral Pneumonia-246.png,True,G20111,test,39bd85eb4af350aedb60c3641d737e21,d42ea33ec20b92eb
Viral Pneumonia-247.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-247.png,Viral Pneumonia/masks/Viral Pneumonia-247.png,True,G20112,test,aa09b17bee2a2175311b1856ec917469,903ce5433fb46b86
Viral Pneumonia-248.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-248.png,Viral Pneumonia/masks/Viral Pneumonia-248.png,True,G20113,test,712612abf127659126d6ab7ae6d7f7f7,d0a5c31fc469b46d
Viral Pneumonia-249.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-249.png,Viral Pneumonia/masks/Viral Pneumonia-249.png,True,G20114,train,60cf0049b29d4ee1c0743b969a4b2130,d03b7a5a5a5a1b1a
Viral Pneumonia-25.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-25.png,Viral Pneumonia/masks/Viral Pneumonia-25.png,True,G20115,train,77926d4a3482d508490ee8fdb624be32,912dc45ed34e312f
Viral Pneumonia-250.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-250.png,Viral Pneumonia/masks/Viral Pneumonia-250.png,True,G20114,train,60cf0049b29d4ee1c0743b969a4b2130,d03b7a5a5a5a1b1a
Viral Pneumonia-251.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-251.png,Viral Pneumonia/masks/Viral Pneumonia-251.png,True,G20116,train,660c1ca69c2bf4ee2848fedcaee8fb1b,947aeb3ee01640d7
Viral Pneumonia-252.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-252.png,Viral Pneumonia/masks/Viral Pneumonia-252.png,True,G20117,train,79b2669a52a3dde6a32a9f6787a90cf6,d02de22ffa683c68
Viral Pneumonia-253.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-253.png,Viral Pneumonia/masks/Viral Pneumonia-253.png,True,G20118,train,685cbc89300dd95b4e4d4d01057bc126,d4ef870b852d6907
Viral Pneumonia-254.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-254.png,Viral Pneumonia/masks/Viral Pneumonia-254.png,True,G20119,test,38ddf0ba2d0ced20a565905355f2c2de,d520c08baf783cfc
Viral Pneumonia-255.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-255.png,Viral Pneumonia/masks/Viral Pneumonia-255.png,True,G20120,train,1872fcad19a3595b9eb8f8b54437a5be,d06a2f78bc3c6466
Viral Pneumonia-256.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-256.png,Viral Pneumonia/masks/Viral Pneumonia-256.png,True,G20121,train,bbb95c5ef82dee6f95dd1909525091a4,d00fc70f9fc32b90
Viral Pneumonia-257.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-257.png,Viral Pneumonia/masks/Viral Pneumonia-257.png,True,G20122,train,96dfb72f7eec6683aa242379b641ce3f,d07ee70bbd8b2904
Viral Pneumonia-258.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-258.png,Viral Pneumonia/masks/Viral Pneumonia-258.png,True,G20123,train,f0b759cca6e0b3a9f66797fff736b72e,d47ee30b83293c87
Viral Pneumonia-259.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-259.png,Viral Pneumonia/masks/Viral Pneumonia-259.png,True,G20124,train,2fa36dc21b7d17708d600151cf88c2ae,d119a5416bb5e2cb
Viral Pneumonia-26.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-26.png,Viral Pneumonia/masks/Viral Pneumonia-26.png,True,G20125,train,4287e37b3c3c28777de04f2c883c3115,94a0e31ed25bf44b
Viral Pneumonia-260.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-260.png,Viral Pneumonia/masks/Viral Pneumonia-260.png,True,G20126,train,0ee33608435c586a64da7d21cadb6452,d062865ac13d79b7
Viral Pneumonia-261.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-261.png,Viral Pneumonia/masks/Viral Pneumonia-261.png,True,G20127,train,71298817af2d7805499d3681d26d5a79,916de63abe5a3c08
Viral Pneumonia-262.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-262.png,Viral Pneumonia/masks/Viral Pneumonia-262.png,True,G20128,train,be0c3a6d7e4ba7adc61aa08f8bc33f34,d4aeab3b8f1c440d
Viral Pneumonia-263.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-263.png,Viral Pneumonia/masks/Viral Pneumonia-263.png,True,G20129,val,80866503419a97da281aa5d7ff1713c5,d0f4c77bb539a102
Viral Pneumonia-264.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-264.png,Viral Pneumonia/masks/Viral Pneumonia-264.png,True,G20130,val,cd4783cbca4b4c74c90a44a13d31342f,d46a2f3ea33c283c
Viral Pneumonia-265.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-265.png,Viral Pneumonia/masks/Viral Pneumonia-265.png,True,G20131,train,107d4a9fad41f7f4e9c8171c84b91ca2,d4aa5757425a5752
Viral Pneumonia-266.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-266.png,Viral Pneumonia/masks/Viral Pneumonia-266.png,True,G20132,train,b2d8f0f5eca04630f27f4665d1a51263,81a1e63ae8ce78d6
Viral Pneumonia-267.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-267.png,Viral Pneumonia/masks/Viral Pneumonia-267.png,True,G20133,train,763338a227cfcd2142abc6f1810203c3,d0ffaf3f2818242c
Viral Pneumonia-268.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-268.png,Viral Pneumonia/masks/Viral Pneumonia-268.png,True,G20134,train,638fdff1dd073a9d9e07e36cb6a18269,d52ac35b554b1f42
Viral Pneumonia-269.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-269.png,Viral Pneumonia/masks/Viral Pneumonia-269.png,True,G20135,train,627941bc094e56bc18ff991401246701,8035c39eeb3bb449
Viral Pneumonia-27.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-27.png,Viral Pneumonia/masks/Viral Pneumonia-27.png,True,G20136,train,bd9b605072f09c88abea416229863f97,d46aa71bcb396138
Viral Pneumonia-270.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-270.png,Viral Pneumonia/masks/Viral Pneumonia-270.png,True,G20137,train,e07efdb7048041bd30b105fdbbfb6824,8167965a855aa779
Viral Pneumonia-271.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-271.png,Viral Pneumonia/masks/Viral Pneumonia-271.png,True,G20138,train,85752e8ede478c832e677ed4315a3c41,9135c61a90cb3dfc
Viral Pneumonia-272.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-272.png,Viral Pneumonia/masks/Viral Pneumonia-272.png,True,G20139,train,ae9a21a3f7f8e57bfc4a509bb1780fa5,90edc63acf39e029
Viral Pneumonia-273.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-273.png,Viral Pneumonia/masks/Viral Pneumonia-273.png,True,G20140,train,51c1b800d37979c25a153bd2eb6e3ea9,d475e31fad3c240c
Viral Pneumonia-274.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-274.png,Viral Pneumonia/masks/Viral Pneumonia-274.png,True,G20141,train,8a29c5f0b3b106084d55b93a848ab07d,d062861ac139fd6f
Viral Pneumonia-275.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-275.png,Viral Pneumonia/masks/Viral Pneumonia-275.png,True,G20142,val,03aa36dc9b494666aa667625dd4a9ec8,d036f30bd70dbcc0
Viral Pneumonia-276.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-276.png,Viral Pneumonia/masks/Viral Pneumonia-276.png,True,G20143,train,2bd1e7ecc90f4a89886ae5ab70225d58,9025c61ac31bbd6f
Viral Pneumonia-277.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-277.png,Viral Pneumonia/masks/Viral Pneumonia-277.png,True,G20144,val,a407b3f6fa1c3136bd1d452f8612c62f,9029e21ee839fd69
Viral Pneumonia-278.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-278.png,Viral Pneumonia/masks/Viral Pneumonia-278.png,True,G20145,test,ccf70e356ecfe417ed38b250e7fb2710,d0afe21fc74b1c48
Viral Pneumonia-279.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-279.png,Viral Pneumonia/masks/Viral Pneumonia-279.png,True,G20146,test,e6f66133f966b90320e4db28a837c1a1,d025c61ec10dfc6f
Viral Pneumonia-28.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-28.png,Viral Pneumonia/masks/Viral Pneumonia-28.png,True,G20147,train,4dbeddd464bf8b718def5ee18e87fce0,9460861ec3bf7987
Viral Pneumonia-280.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-280.png,Viral Pneumonia/masks/Viral Pneumonia-280.png,True,G20148,test,1b2d39327879e1e40c147a56174b216a,90ade61ae27ba06b
Viral Pneumonia-281.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-281.png,Viral Pneumonia/masks/Viral Pneumonia-281.png,True,G20149,train,9cc0165360e9fff125b4464981013e85,9007c31eea0ff84f
Viral Pneumonia-282.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-282.png,Viral Pneumonia/masks/Viral Pneumonia-282.png,True,G20150,train,06919e538d6b67e293ff8b44c30fcf2d,d414ebc76bbc60c2
Viral Pneumonia-283.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-283.png,Viral Pneumonia/masks/Viral Pneumonia-283.png,True,G20151,val,d009816ebc19b15304f3026da6fbedbe,d4b4e31bc2cb9cc2
Viral Pneumonia-284.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-284.png,Viral Pneumonia/masks/Viral Pneumonia-284.png,True,G20152,train,d897ebb8f8b5d6b48b2d785cb3039970,81e0863aa43af97f
Viral Pneumonia-285.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-285.png,Viral Pneumonia/masks/Viral Pneumonia-285.png,True,G20153,val,2596999425488ddfa8eba62e6abf37db,c13dd20b904ffce2
Viral Pneumonia-286.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-286.png,Viral Pneumonia/masks/Viral Pneumonia-286.png,True,G20154,train,a63d5f8226e0d02a3bc47a7d7e06dc4e,d07de71ea90f0945
Viral Pneumonia-287.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-287.png,Viral Pneumonia/masks/Viral Pneumonia-287.png,True,G20155,train,a08cd4bbb6014d5617e84ad22b0cd22a,9164861aa139fd6f
Viral Pneumonia-288.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-288.png,Viral Pneumonia/masks/Viral Pneumonia-288.png,True,G20156,train,81bd6fceb125126c29f2de1131e7919e,913fd418b171b5cc
Viral Pneumonia-289.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-289.png,Viral Pneumonia/masks/Viral Pneumonia-289.png,True,G20157,train,7ef15a1307b604c88a3c6146c477da36,956ae63abaca2aa8
Viral Pneumonia-29.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-29.png,Viral Pneumonia/masks/Viral Pneumonia-29.png,True,G20158,train,4fde48a1fe8fbb132f5731884863ef29,917ee6968a973a90
Viral Pneumonia-290.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-290.png,Viral Pneumonia/masks/Viral Pneumonia-290.png,True,G20159,train,9a339fdf56feb30651a4d1187269153d,d06dc73ee92c3c48
Viral Pneumonia-291.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-291.png,Viral Pneumonia/masks/Viral Pneumonia-291.png,True,G20160,train,f077a44e7fc807c5971984b6060bd5bb,9125c61ac57b3c67
Viral Pneumonia-292.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-292.png,Viral Pneumonia/masks/Viral Pneumonia-292.png,True,G20161,train,87ae949f1673e09480e354ea9520686f,d07ec61a98c93dd4
Viral Pneumonia-293.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-293.png,Viral Pneumonia/masks/Viral Pneumonia-293.png,True,G20162,train,28f48c4a62e5b37ab670d94a56b69b99,912fc31e8e693ce2
Viral Pneumonia-294.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-294.png,Viral Pneumonia/masks/Viral Pneumonia-294.png,True,G20163,val,a50d6fe17090c2a3741d2627446374a1,912bc21e595d5d59
Viral Pneumonia-295.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-295.png,Viral Pneumonia/masks/Viral Pneumonia-295.png,True,G20163,val,a50d6fe17090c2a3741d2627446374a1,912bc21e595d5d59
Viral Pneumonia-296.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-296.png,Viral Pneumonia/masks/Viral Pneumonia-296.png,True,G20164,train,e2edf92b462b4ccb974e37bb10b3dac1,dd28c20f5f493d68
Viral Pneumonia-297.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-297.png,Viral Pneumonia/masks/Viral Pneumonia-297.png,True,G20165,train,bc6b761cd60b13438b2d6670339b5bfd,942f031e0b7c6d7c
Viral Pneumonia-298.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-298.png,Viral Pneumonia/masks/Viral Pneumonia-298.png,True,G20166,train,45c27c584bfacc12015071853bcbc32a,9428e2073df1873f
Viral Pneumonia-299.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-299.png,Viral Pneumonia/masks/Viral Pneumonia-299.png,True,G20167,val,5d9b89f63da2b8cec62770ba30dfda2b,946ac3128369bde7
Viral Pneumonia-3.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-3.png,Viral Pneumonia/masks/Viral Pneumonia-3.png,True,G20168,test,9bd60761b0f92f3978db32c93d5cd74f,912bea78e959390b
Viral Pneumonia-30.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-30.png,Viral Pneumonia/masks/Viral Pneumonia-30.png,True,G20169,test,6472ddd09740f1277876493e7063bd77,d4bfe30f440d1c4b
Viral Pneumonia-300.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-300.png,Viral Pneumonia/masks/Viral Pneumonia-300.png,True,G20170,test,46051e3902378d1003e1165d0d8a8e86,906d287c2ae9eaec
Viral Pneumonia-301.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-301.png,Viral Pneumonia/masks/Viral Pneumonia-301.png,True,G20171,train,59bf74b040944a23eec2ea1cc48af767,91656b3dc23cc61e
Viral Pneumonia-302.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-302.png,Viral Pneumonia/masks/Viral Pneumonia-302.png,True,G20172,train,71672b8f7fdb9948e5f9e882c359ad04,982de13fe40f3cc8
Viral Pneumonia-303.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-303.png,Viral Pneumonia/masks/Viral Pneumonia-303.png,True,G20173,train,6c4a823374004ab87b6f34d25a2cf4df,956a2f20872c6b3f
Viral Pneumonia-304.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-304.png,Viral Pneumonia/masks/Viral Pneumonia-304.png,True,G20174,train,ba74ab09c164bafec16c376a1379f690,942fab1b9b4b1c54
Viral Pneumonia-305.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-305.png,Viral Pneumonia/masks/Viral Pneumonia-305.png,True,G20175,train,8a1ca1ef211e253d4dd01748a3534afe,d42b6b2e1c4b3d1c
Viral Pneumonia-306.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-306.png,Viral Pneumonia/masks/Viral Pneumonia-306.png,True,G20176,train,f29f2c7553ba041659b56fbb160c5d9a,d46f6f0e0f4b2c48
Viral Pneumonia-307.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-307.png,Viral Pneumonia/masks/Viral Pneumonia-307.png,True,G20177,train,9fb7d7bb79673d721063aa204ff8cacd,d16e2fb21b0d1e1c
Viral Pneumonia-308.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-308.png,Viral Pneumonia/masks/Viral Pneumonia-308.png,True,G20178,train,0d94ffcb64de2c0aa7a7a5b436094288,d46be33e836b3860
Viral Pneumonia-309.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-309.png,Viral Pneumonia/masks/Viral Pneumonia-309.png,True,G20179,train,3f4e1a663e309515ac58951eb55361e5,9059e56f69864e35
Viral Pneumonia-31.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-31.png,Viral Pneumonia/masks/Viral Pneumonia-31.png,True,G20180,train,aa9548886626c18ff5d8bbf154ead47d,c83dc31919593dd9
Viral Pneumonia-310.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-310.png,Viral Pneumonia/masks/Viral Pneumonia-310.png,True,G20181,train,d2adfdef856261037a8a18f01f493af1,d17fc63ac8d13990
Viral Pneumonia-311.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-311.png,Viral Pneumonia/masks/Viral Pneumonia-311.png,True,G20182,train,eb83122e15e4ee7e2ebc5f12939aae19,d1fcc61a854bb0cb
Viral Pneumonia-312.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-312.png,Viral Pneumonia/masks/Viral Pneumonia-312.png,True,G20183,train,5e41470d8c264c6ac8916bc5fe6244d1,d4e6873b871ea04e
Viral Pneumonia-313.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-313.png,Viral Pneumonia/masks/Viral Pneumonia-313.png,True,G20184,train,6d4696f69944972106b5a59129798d45,99ffe63a820a38ca
Viral Pneumonia-314.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-314.png,Viral Pneumonia/masks/Viral Pneumonia-314.png,True,G20185,val,668da0e3e76a1e89135f0245e1bac8fc,d428f20d1f6d0c7e
Viral Pneumonia-315.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-315.png,Viral Pneumonia/masks/Viral Pneumonia-315.png,True,G20186,val,825a79452c95321ed627e38d1a6cf32f,d12a077b7869585d
Viral Pneumonia-316.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-316.png,Viral Pneumonia/masks/Viral Pneumonia-316.png,True,G20187,test,b188f7031e6b60721714b054cd02f6e7,d429534f0f492d3e
Viral Pneumonia-317.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-317.png,Viral Pneumonia/masks/Viral Pneumonia-317.png,True,G20188,train,1a0811409f8d2a3e1320d288ae567218,816b677656525a5a
Viral Pneumonia-318.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-318.png,Viral Pneumonia/masks/Viral Pneumonia-318.png,True,G20189,val,191b0a49ec73e0b387d3d21aff7c7cf8,9018e3633de4cfa6
Viral Pneumonia-319.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-319.png,Viral Pneumonia/masks/Viral Pneumonia-319.png,True,G20190,train,92adcf8cbb06849605078f41b6e2f03e,dd75e03bc20fb00b
Viral Pneumonia-32.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-32.png,Viral Pneumonia/masks/Viral Pneumonia-32.png,True,G20191,train,3b7f081866ba62b89eab013b1d170cb4,9125c63ac679d84f
Viral Pneumonia-320.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-320.png,Viral Pneumonia/masks/Viral Pneumonia-320.png,True,G20192,test,f68b2f500d9dfa8201eba446137df42f,810ba747373ee887
Viral Pneumonia-321.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-321.png,Viral Pneumonia/masks/Viral Pneumonia-321.png,True,G20193,test,bfc4ae9d5ac5e65a94c3c97b2af15734,9125c73ab84b39c7
Viral Pneumonia-322.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-322.png,Viral Pneumonia/masks/Viral Pneumonia-322.png,True,G20194,train,130dbe527cb9794d1872c9b63b0b3c2c,912b571e5d491d1d
Viral Pneumonia-323.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-323.png,Viral Pneumonia/masks/Viral Pneumonia-323.png,True,G20195,train,d271c65ca22f3d24d79fe7b1d8017f3b,99a1eb3cc73cc00f
Viral Pneumonia-324.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-324.png,Viral Pneumonia/masks/Viral Pneumonia-324.png,True,G20196,train,35c6284a3c8625f47b31aae450ffbf97,81bee71e49475847
Viral Pneumonia-325.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-325.png,Viral Pneumonia/masks/Viral Pneumonia-325.png,True,G20197,train,bbe3c20f0a8bf908a630c77958a73a3e,d4aaa31fc55dc42c
Viral Pneumonia-326.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-326.png,Viral Pneumonia/masks/Viral Pneumonia-326.png,True,G20198,train,9618988507408219073da012273cfa47,c13fc219b8693cda
Viral Pneumonia-327.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-327.png,Viral Pneumonia/masks/Viral Pneumonia-327.png,True,G20199,train,ca41a38529a86412af616dbc58e355bc,8937c616d44eb4ea
Viral Pneumonia-328.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-328.png,Viral Pneumonia/masks/Viral Pneumonia-328.png,True,G20200,train,a638ea625e3f1b101bd5287dfe1e1073,84f4a31bc77b864a
Viral Pneumonia-329.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-329.png,Viral Pneumonia/masks/Viral Pneumonia-329.png,True,G20201,train,cc728bf0dde57840f0eb68144051e9eb,90f5871ac63e6987
Viral Pneumonia-33.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-33.png,Viral Pneumonia/masks/Viral Pneumonia-33.png,True,G20202,train,3e1d42002a829631bc5a1d0ee245dedc,d47f0a195f5d0a0e
Viral Pneumonia-330.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-330.png,Viral Pneumonia/masks/Viral Pneumonia-330.png,True,G20203,train,7525e743d8d92b1aecc12a595024e6e1,9039e15717c217f6
Viral Pneumonia-331.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-331.png,Viral Pneumonia/masks/Viral Pneumonia-331.png,True,G20204,val,c702a94206cc8319cb97a8d6c2679f8b,912be75e79783860
Viral Pneumonia-332.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-332.png,Viral Pneumonia/masks/Viral Pneumonia-332.png,True,G20205,train,ec175a1a98b3bae4e03a5ea8adebf07b,807be75e3ec01ab8
Viral Pneumonia-333.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-333.png,Viral Pneumonia/masks/Viral Pneumonia-333.png,True,G20206,val,7de723933979b575929ebc7c5b9bb124,8127c61e9e4b3dc9
Viral Pneumonia-334.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-334.png,Viral Pneumonia/masks/Viral Pneumonia-334.png,True,G20207,val,1473d975873e9bffaa60c878d468affa,9123c33ecc6b34ce
Viral Pneumonia-335.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-335.png,Viral Pneumonia/masks/Viral Pneumonia-335.png,True,G20208,test,a9a6e16f26c38f269308eb902f5ef6a1,d5286ce2232b6ee6
Viral Pneumonia-336.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-336.png,Viral Pneumonia/masks/Viral Pneumonia-336.png,True,G20209,train,d7301873ab6127797bdeae0d7c5e0f89,d1e2c73ac729906d
Viral Pneumonia-337.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-337.png,Viral Pneumonia/masks/Viral Pneumonia-337.png,True,G20210,train,e5587130513e9c64c353e40ecea6abf2,903bd71e8c943db4
Viral Pneumonia-338.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-338.png,Viral Pneumonia/masks/Viral Pneumonia-338.png,True,G20211,train,05070209e809ca4d82586210343ca92d,d4ef8309c51619d7
Viral Pneumonia-339.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-339.png,Viral Pneumonia/masks/Viral Pneumonia-339.png,True,G20212,test,ffa819e402871326d96657d7b0b89276,9c6ea33a8b973885
Viral Pneumonia-34.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-34.png,Viral Pneumonia/masks/Viral Pneumonia-34.png,True,G20213,test,da85f4d8f3e3f570d185a8d7cb19d5d0,9074c61ac73b917b
Viral Pneumonia-340.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-340.png,Viral Pneumonia/masks/Viral Pneumonia-340.png,True,G20214,train,820c147e41b1e09946de6201293a9a1f,d027961b956cb5cc
Viral Pneumonia-341.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-341.png,Viral Pneumonia/masks/Viral Pneumonia-341.png,True,G20215,train,3a555b3a0b0046373d278e131261ca67,946e831b8b3c6c6e
Viral Pneumonia-342.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-342.png,Viral Pneumonia/masks/Viral Pneumonia-342.png,True,G20216,test,4ffeac9ef77b8b8ab09b3adf0bd06aad,946ec559c5c4cacb
Viral Pneumonia-343.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-343.png,Viral Pneumonia/masks/Viral Pneumonia-343.png,True,G20217,train,70da33946ace4b1762d47b6e1125e452,8d1c64e5d63e6c92
Viral Pneumonia-344.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-344.png,Viral Pneumonia/masks/Viral Pneumonia-344.png,True,G20218,val,6f2580efc3c95df167b37536d9de45f1,946e2e0eaf3cc62c
Viral Pneumonia-345.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-345.png,Viral Pneumonia/masks/Viral Pneumonia-345.png,True,G20219,val,18e22f5d056bea201d88b8194982e7c2,c07a67473c94c70f
Viral Pneumonia-346.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-346.png,Viral Pneumonia/masks/Viral Pneumonia-346.png,True,G20220,train,facd3d2ecca8d5539b40e53ac2b51d6f,d07b470c2c2f3c6d
Viral Pneumonia-347.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-347.png,Viral Pneumonia/masks/Viral Pneumonia-347.png,True,G20221,train,57a6776943e883b3ff1fbb8b855c979c,d02d65071e4a4b7f
Viral Pneumonia-348.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-348.png,Viral Pneumonia/masks/Viral Pneumonia-348.png,True,G20222,val,2ba5868de4a76e2dff6a537c648368e0,906bc71eda4b5945
Viral Pneumonia-349.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-349.png,Viral Pneumonia/masks/Viral Pneumonia-349.png,True,G20223,test,4597988ee208de0b2f86e23bbf899d7e,d56b671e1c432b25
Viral Pneumonia-35.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-35.png,Viral Pneumonia/masks/Viral Pneumonia-35.png,True,G20224,train,3ccf86d958a7c7e8bcc0c84b33f97227,8121fa12eb3afc4b
Viral Pneumonia-350.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-350.png,Viral Pneumonia/masks/Viral Pneumonia-350.png,True,G20225,train,8c86aaa0071e403cc3ffd6eff2df8426,907bc71c1c5b1e0f
Viral Pneumonia-351.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-351.png,Viral Pneumonia/masks/Viral Pneumonia-351.png,True,G20226,train,4759a81f49f000e7b4ea5b84559b686d,c03fea3e38eb2c28
Viral Pneumonia-352.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-352.png,Viral Pneumonia/masks/Viral Pneumonia-352.png,True,G20227,train,920a8577fdc70f7a7e3a1445c81f641e,d06b633e3e535f00
Viral Pneumonia-353.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-353.png,Viral Pneumonia/masks/Viral Pneumonia-353.png,True,G20228,train,72526b50158a3567b3baac03f17a0e3e,803bc54ff54a355a
Viral Pneumonia-354.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-354.png,Viral Pneumonia/masks/Viral Pneumonia-354.png,True,G20229,train,982feff7ca5bc517a438bf776c239b6f,996cc63ac73b3066
Viral Pneumonia-355.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-355.png,Viral Pneumonia/masks/Viral Pneumonia-355.png,True,G20230,test,026d5a9e0402cda0518c51a33a566983,d47ca21fa30b1b2d
Viral Pneumonia-356.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-356.png,Viral Pneumonia/masks/Viral Pneumonia-356.png,True,G20231,train,cf6ae4db934b0e17f99cdc3c26403540,957eaa3eba1a181c
Viral Pneumonia-357.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-357.png,Viral Pneumonia/masks/Viral Pneumonia-357.png,True,G20232,test,1fb645920e6a82c1d1fac72833958cc0,d42f2f07054f6d4c
Viral Pneumonia-358.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-358.png,Viral Pneumonia/masks/Viral Pneumonia-358.png,True,G20233,train,3b0f6d0c2e7c779cc4c1b2160830e96b,d57f2f161a092d2c
Viral Pneumonia-359.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-359.png,Viral Pneumonia/masks/Viral Pneumonia-359.png,True,G20234,train,405b237fd273642f0f28abf6ea4cdada,917fa7161a875b48
Viral Pneumonia-36.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-36.png,Viral Pneumonia/masks/Viral Pneumonia-36.png,True,G20235,train,e5ae51265eb33fcea8245427f926a84d,81a7ea1be939c54a
Viral Pneumonia-360.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-360.png,Viral Pneumonia/masks/Viral Pneumonia-360.png,True,G20236,train,ad8f0de9acf613bd0f35bc8061b65666,957e6f46a6171215
Viral Pneumonia-361.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-361.png,Viral Pneumonia/masks/Viral Pneumonia-361.png,True,G20237,train,09188615c25223bb3955b1bbf6d0b6ff,916be71ed04a9c5a
Viral Pneumonia-362.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-362.png,Viral Pneumonia/masks/Viral Pneumonia-362.png,True,G20238,train,2d6d0dc67fe60b79b931986f555883b6,d46e0705155f6c7c
Viral Pneumonia-363.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-363.png,Viral Pneumonia/masks/Viral Pneumonia-363.png,True,G20239,train,d7e3fd655ca392dbdaef6c4d6b2e9f40,d462c77a851e29cd
Viral Pneumonia-364.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-364.png,Viral Pneumonia/masks/Viral Pneumonia-364.png,True,G20240,train,f600e8a88b3bb7ff3e9af37798fd9f57,8139e517a5ce3f90
Viral Pneumonia-365.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-365.png,Viral Pneumonia/masks/Viral Pneumonia-365.png,True,G20241,train,58d7955244b49fc0d9e7f2d554cdc7c2,9197c61ae9a935c9
Viral Pneumonia-366.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-366.png,Viral Pneumonia/masks/Viral Pneumonia-366.png,True,G20242,train,79ad727893eb2078f1e153988a738331,902be71ec64f1c47
Viral Pneumonia-367.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-367.png,Viral Pneumonia/masks/Viral Pneumonia-367.png,True,G20243,train,f3404b143b8da2bd9fcb537a7cb3d566,91fea71a830f1987
Viral Pneumonia-368.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-368.png,Viral Pneumonia/masks/Viral Pneumonia-368.png,True,G20244,train,d0deae3f4c8f157876e70060c84512d2,d02bc31ecc599ce9
Viral Pneumonia-369.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-369.png,Viral Pneumonia/masks/Viral Pneumonia-369.png,True,G20245,train,361141e052cf4810fccb733102974119,d025e33c6939786b
Viral Pneumonia-37.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-37.png,Viral Pneumonia/masks/Viral Pneumonia-37.png,True,G20246,val,0acb304192fc840426305ac5b1eb0749,d1a5e61ae40fbcc1
Viral Pneumonia-370.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-370.png,Viral Pneumonia/masks/Viral Pneumonia-370.png,True,G20247,val,b1fe1a4341d6f2e4e90ea6537b60b1c2,9c6fe31fc13c242c
Viral Pneumonia-371.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-371.png,Viral Pneumonia/masks/Viral Pneumonia-371.png,True,G20248,val,f7255eb89aac91013601d4a2aff45621,987b471e1a2b3ca9
Viral Pneumonia-372.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-372.png,Viral Pneumonia/masks/Viral Pneumonia-372.png,True,G20249,test,1f2f199433e3dd05a655031137e1cd47,9436c31ff24d30cb
Viral Pneumonia-373.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-373.png,Viral Pneumonia/masks/Viral Pneumonia-373.png,True,G20250,train,d98b6c924fb63cb5d83e58044a5e3fe7,996da61a05d33d96
Viral Pneumonia-374.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-374.png,Viral Pneumonia/masks/Viral Pneumonia-374.png,True,G20251,train,d566e3904185579e80716217e4044f7e,dd2a43174b1b1eaa
Viral Pneumonia-375.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-375.png,Viral Pneumonia/masks/Viral Pneumonia-375.png,True,G20252,test,88a9c4311dc5141675428bed3ad5ef02,907bc31e4a5b1d4d
Viral Pneumonia-376.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-376.png,Viral Pneumonia/masks/Viral Pneumonia-376.png,True,G20253,train,57e587eadd5f59c133d646b96856316c,913dc61bd24b38d3
Viral Pneumonia-377.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-377.png,Viral Pneumonia/masks/Viral Pneumonia-377.png,True,G20254,train,e2c45d05f9feb96b5b6799b2fd233bd0,9039e74a25c66fb4
Viral Pneumonia-378.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-378.png,Viral Pneumonia/masks/Viral Pneumonia-378.png,True,G20255,train,665543094590e481198fd431894edb75,d06fef0f0d49382a
Viral Pneumonia-379.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-379.png,Viral Pneumonia/masks/Viral Pneumonia-379.png,True,G20256,train,7387bff89e1b2c4e397290228759a9fa,d47ce31f9f420aa4
Viral Pneumonia-38.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-38.png,Viral Pneumonia/masks/Viral Pneumonia-38.png,True,G20257,val,177a3bf3b329f3b8ccfdb117d4f567ba,d16e4e4b0d2e3936
Viral Pneumonia-380.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-380.png,Viral Pneumonia/masks/Viral Pneumonia-380.png,True,G20258,train,a4851fc32377dfcc8c8ce974a0917fff,c43b641d5d4f1a6c
Viral Pneumonia-381.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-381.png,Viral Pneumonia/masks/Viral Pneumonia-381.png,True,G20259,val,400fca43a462bdb1bdde7df54a2666c0,943ec61aeb8f3984
Viral Pneumonia-382.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-382.png,Viral Pneumonia/masks/Viral Pneumonia-382.png,True,G20260,train,8f2a71558315ca906bff9d83c1d99c94,8125e62abcc2bfb1
Viral Pneumonia-383.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-383.png,Viral Pneumonia/masks/Viral Pneumonia-383.png,True,G20261,train,448396d9ff7656b46d69da23901f1760,d07ce61a9ac34bb8
Viral Pneumonia-384.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-384.png,Viral Pneumonia/masks/Viral Pneumonia-384.png,True,G20262,val,793519d89ea71f35a346e8425a96325c,9174c31acd7dc4c6
Viral Pneumonia-385.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-385.png,Viral Pneumonia/masks/Viral Pneumonia-385.png,True,G20263,train,60ad4fc25115f6552232c437066ba8ac,913be5c67c90ca5a
Viral Pneumonia-386.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-386.png,Viral Pneumonia/masks/Viral Pneumonia-386.png,True,G20264,test,4f37a32c4e50f349cc550be30334544f,d06fce1e194d1ccc
Viral Pneumonia-387.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-387.png,Viral Pneumonia/masks/Viral Pneumonia-387.png,True,G20265,train,4338971cd06e9e6a28573fddf30c6cbb,907e47161a4b1f4f
Viral Pneumonia-388.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-388.png,Viral Pneumonia/masks/Viral Pneumonia-388.png,True,G20266,train,cf3cf9d727218877839de478f77c4d08,9939e1e53c94cab4
Viral Pneumonia-389.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-389.png,Viral Pneumonia/masks/Viral Pneumonia-389.png,True,G20267,train,29e6aebdf493dd4dc6e476c27a2be12a,9929e1073fd4c60f
Viral Pneumonia-39.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-39.png,Viral Pneumonia/masks/Viral Pneumonia-39.png,True,G20268,train,2fc1cb495bd9cf978b6b52852f6b6675,d45f471e194d3c46
Viral Pneumonia-390.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-390.png,Viral Pneumonia/masks/Viral Pneumonia-390.png,True,G20269,test,044fd8546584af1bd0ef037f661c1113,c039e5c76e9cc4c3
Viral Pneumonia-391.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-391.png,Viral Pneumonia/masks/Viral Pneumonia-391.png,True,G20270,test,b00b571147e186b86b1b04966c4fc005,dd34e00bb54a3dd2
Viral Pneumonia-392.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-392.png,Viral Pneumonia/masks/Viral Pneumonia-392.png,True,G20271,train,e63b563c5f36c0bfa06adaf5825f9ce1,d02de33869693ccd
Viral Pneumonia-393.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-393.png,Viral Pneumonia/masks/Viral Pneumonia-393.png,True,G20272,train,176a973db8790140f686a57ee75f6940,916fc63ae8091ccf
Viral Pneumonia-394.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-394.png,Viral Pneumonia/masks/Viral Pneumonia-394.png,True,G20273,train,6519809db686aaf82370f38f7bf302d2,d0afc61a854a3d4f
Viral Pneumonia-395.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-395.png,Viral Pneumonia/masks/Viral Pneumonia-395.png,True,G20274,train,6af8ed98bbc56feb4f48bc1840db793c,d07ce73ec10339c3
Viral Pneumonia-396.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-396.png,Viral Pneumonia/masks/Viral Pneumonia-396.png,True,G20275,train,4531aba8f6a9998ff8c3c13de99f408f,812fc21ef87939d1
Viral Pneumonia-397.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-397.png,Viral Pneumonia/masks/Viral Pneumonia-397.png,True,G20276,train,ce0f073bbd9fd416b9331f0221b865de,d47dc71e85293c0e
Viral Pneumonia-398.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-398.png,Viral Pneumonia/masks/Viral Pneumonia-398.png,True,G20277,val,419657e9b13729e4dc6d4a874f1a79a6,98f1e73ea44e300f
Viral Pneumonia-399.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-399.png,Viral Pneumonia/masks/Viral Pneumonia-399.png,True,G20278,train,f0fda73281247133b06ba8a39412a84f,98a1e77f944238d5
Viral Pneumonia-4.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-4.png,Viral Pneumonia/masks/Viral Pneumonia-4.png,True,G20279,train,5a0fad4960b76a95ee0cc4d49d4f3882,d025c61ef98b1fc2
Viral Pneumonia-40.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-40.png,Viral Pneumonia/masks/Viral Pneumonia-40.png,True,G20280,train,2eceea8515f8f8703f999305745bc652,d06bc73ec16d3c44
Viral Pneumonia-400.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-400.png,Viral Pneumonia/masks/Viral Pneumonia-400.png,True,G20281,train,79b4adc591721b141ee8f9293381a4cd,d035c21bbc693cf8
Viral Pneumonia-401.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-401.png,Viral Pneumonia/masks/Viral Pneumonia-401.png,True,G20282,train,e8ac5195ae4f0666f47c2a6e09d5ea7b,916bde595d591816
Viral Pneumonia-402.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-402.png,Viral Pneumonia/masks/Viral Pneumonia-402.png,True,G20283,train,10a86ff24ddea4c58aaf2fdb3e082d29,911fce1ab14f34c6
Viral Pneumonia-403.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-403.png,Viral Pneumonia/masks/Viral Pneumonia-403.png,True,G20284,train,d4d59c55636717f74f391e16fea406f6,90ef871a871ac15f
Viral Pneumonia-404.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-404.png,Viral Pneumonia/masks/Viral Pneumonia-404.png,True,G20285,val,8096b9680618c5480ced145c3c1c1576,8135c21ede4b3cc7
Viral Pneumonia-405.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-405.png,Viral Pneumonia/masks/Viral Pneumonia-405.png,True,G20286,train,483d582e221e8c25d180c944fb6ec6ba,8137c31eec5a3ce1
Viral Pneumonia-406.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-406.png,Viral Pneumonia/masks/Viral Pneumonia-406.png,True,G20287,val,e6ab9d83fac318c350a94cb368a3d070,d064c61a956db54f
Viral Pneumonia-407.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-407.png,Viral Pneumonia/masks/Viral Pneumonia-407.png,True,G20288,test,5a37e923d2f34991a05d911a208d384c,91fdf639a53ca00c
Viral Pneumonia-408.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-408.png,Viral Pneumonia/masks/Viral Pneumonia-408.png,True,G20289,test,445684d3b6bd05ebe6fa759d0d174378,88f4a33a875ae197
Viral Pneumonia-409.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-409.png,Viral Pneumonia/masks/Viral Pneumonia-409.png,True,G20290,train,53f072db1c6499d5ff10369cf615344b,d46aab018775c7aa
Viral Pneumonia-41.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-41.png,Viral Pneumonia/masks/Viral Pneumonia-41.png,True,G20291,test,310e5da8f0f7f1529884e5c572d62bb9,9c2a515b5d581e9b
Viral Pneumonia-410.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-410.png,Viral Pneumonia/masks/Viral Pneumonia-410.png,True,G20292,train,9c2cd63573208a6aa209a8c3c1460c59,d074c61ac56db46e
Viral Pneumonia-411.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-411.png,Viral Pneumonia/masks/Viral Pneumonia-411.png,True,G20293,train,034b55fedc1829b72aef350b3db2af0a,d065c65a954bb14f
Viral Pneumonia-412.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-412.png,Viral Pneumonia/masks/Viral Pneumonia-412.png,True,G20294,train,963fad6ac820c5763667ee2ca614cbc2,d429e102cf4b2d6f
Viral Pneumonia-413.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-413.png,Viral Pneumonia/masks/Viral Pneumonia-413.png,True,G20295,train,7468f194fe983731fb8094c05c6f7fa0,9065c63ee43bf04b
Viral Pneumonia-414.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-414.png,Viral Pneumonia/masks/Viral Pneumonia-414.png,True,G20296,train,4dde1f05243ff3fd1bf5609e36152b95,9123c21efe4eb447
Viral Pneumonia-415.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-415.png,Viral Pneumonia/masks/Viral Pneumonia-415.png,True,G20297,train,a27a1d34a07c05241f506d6dcbd3f12b,91d7c71bc139b44a
Viral Pneumonia-416.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-416.png,Viral Pneumonia/masks/Viral Pneumonia-416.png,True,G20298,val,cdc33137cffa5d63b8a6b325f0182686,d12de61ae9693c49
Viral Pneumonia-417.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-417.png,Viral Pneumonia/masks/Viral Pneumonia-417.png,True,G20299,train,46cc967c2d8257f430eeb5a3a18b5f6c,912bc21ae25aec6f
Viral Pneumonia-418.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-418.png,Viral Pneumonia/masks/Viral Pneumonia-418.png,True,G20300,train,62d2253548c22e600685218a193d9364,d062c61ecb1dc54f
Viral Pneumonia-419.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-419.png,Viral Pneumonia/masks/Viral Pneumonia-419.png,True,G20301,train,260660d0ace3dcba5832943c09afed1c,9574c30b852cf1cf
Viral Pneumonia-42.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-42.png,Viral Pneumonia/masks/Viral Pneumonia-42.png,True,G20302,train,1d543580fe661c1edeb7f3a25a4723c2,d0ee8729898c69b7
Viral Pneumonia-420.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-420.png,Viral Pneumonia/masks/Viral Pneumonia-420.png,True,G20303,train,b0d9752faa08f017364f039b41bf4052,d424e31fe20f9e4a
Viral Pneumonia-421.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-421.png,Viral Pneumonia/masks/Viral Pneumonia-421.png,True,G20304,train,5a75ad6a5d35021e49a6dbe041eed369,d46ca73f8d282c2e
Viral Pneumonia-422.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-422.png,Viral Pneumonia/masks/Viral Pneumonia-422.png,True,G20305,train,7cbb3b53a5bb4f409fd9b052f0903cde,9829e157bee04ab5
Viral Pneumonia-423.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-423.png,Viral Pneumonia/masks/Viral Pneumonia-423.png,True,G20306,train,ccc6bf667ba1543c52b3f8a4252950c6,8023e76fb4c8bf30
Viral Pneumonia-424.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-424.png,Viral Pneumonia/masks/Viral Pneumonia-424.png,True,G20307,val,68230be893070ecba25c88352e4d5466,d0eaa73d2d8c3c45
Viral Pneumonia-425.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-425.png,Viral Pneumonia/masks/Viral Pneumonia-425.png,True,G20308,val,2bb357a863f7b775521f27356d0d9a12,9121c31fe4793c6d
Viral Pneumonia-426.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-426.png,Viral Pneumonia/masks/Viral Pneumonia-426.png,True,G20309,test,72777b7ef10dd3a25b292bd2f07d04f7,d4e6832f8739830f
Viral Pneumonia-427.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-427.png,Viral Pneumonia/masks/Viral Pneumonia-427.png,True,G20310,test,772d44e596f0bc13c2fc49f2c61e68cc,d46ac31ec3493dc3
Viral Pneumonia-428.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-428.png,Viral Pneumonia/masks/Viral Pneumonia-428.png,True,G20311,test,bfe29226deda906e42c16ae26920ab86,8115e4e37ee89669
Viral Pneumonia-429.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-429.png,Viral Pneumonia/masks/Viral Pneumonia-429.png,True,G20312,val,651ed4bb79015730b0ad8ab2b2ca30d3,d454e0367ea54b78
Viral Pneumonia-43.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-43.png,Viral Pneumonia/masks/Viral Pneumonia-43.png,True,G20313,train,248376a9b1a40ad96f1b42a3f09c6ba9,c866c71fc70d1947
Viral Pneumonia-430.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-430.png,Viral Pneumonia/masks/Viral Pneumonia-430.png,True,G20314,train,9efb4a74355f5f3b8e6fd74f907d7e32,801965c76bbf66c2
Viral Pneumonia-431.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-431.png,Viral Pneumonia/masks/Viral Pneumonia-431.png,True,G20315,train,4501dd55901409ed837486e102f3913a,d02ea71bc35b5c4a
Viral Pneumonia-432.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-432.png,Viral Pneumonia/masks/Viral Pneumonia-432.png,True,G20316,train,0dc67f3f4f5cc3e36a90cc9f1ba1e88b,d062c31fe529b46b
Viral Pneumonia-433.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-433.png,Viral Pneumonia/masks/Viral Pneumonia-433.png,True,G20317,train,91ea669ec4e058ed24ce76e5f0fec527,c41ce412bef11b3d
Viral Pneumonia-434.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-434.png,Viral Pneumonia/masks/Viral Pneumonia-434.png,True,G20318,train,e7e055e0ad123a82f6f72a7b7e469504,912be31ed84b3c69
Viral Pneumonia-435.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-435.png,Viral Pneumonia/masks/Viral Pneumonia-435.png,True,G20319,train,83c614cfcf9bca09beb3518e7453e29f,912bda5a78593959
Viral Pneumonia-436.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-436.png,Viral Pneumonia/masks/Viral Pneumonia-436.png,True,G20320,train,22c08de9ccddc0486befdd85471610da,907ce06a3a70e75e
Viral Pneumonia-437.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-437.png,Viral Pneumonia/masks/Viral Pneumonia-437.png,True,G20321,train,de9156eb2f58603d44501cac4400b96a,c0ef4b5f4a2f200f
Viral Pneumonia-438.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-438.png,Viral Pneumonia/masks/Viral Pneumonia-438.png,True,G20322,train,e39a12cf7d261910eabdc146116f4d4f,9129e71eea529ec2
Viral Pneumonia-439.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-439.png,Viral Pneumonia/masks/Viral Pneumonia-439.png,True,G20323,train,ff7c0e8d5c2ed8f8d5db104bd4a67857,913fe67ae84b3841
Viral Pneumonia-44.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-44.png,Viral Pneumonia/masks/Viral Pneumonia-44.png,True,G20324,train,0edc1c3ee2442358642c9ca899693338,997f461847393d49
Viral Pneumonia-440.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-440.png,Viral Pneumonia/masks/Viral Pneumonia-440.png,True,G20325,train,344c6b15aab85d9532ee90dd34bb72ad,d0e5c33fe11ae138
Viral Pneumonia-441.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-441.png,Viral Pneumonia/masks/Viral Pneumonia-441.png,True,G20326,train,84e94b781ce45bc4f39ff9a8a760b36d,9127c30ae828fcef
Viral Pneumonia-442.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-442.png,Viral Pneumonia/masks/Viral Pneumonia-442.png,True,G20327,test,e4f507682d3f57ca8118823255ecb6c8,952f861a810ebcef
Viral Pneumonia-443.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-443.png,Viral Pneumonia/masks/Viral Pneumonia-443.png,True,G20328,val,d83112010c374916c73d1104bb5b4975,d46fe31fe50d8444
Viral Pneumonia-444.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-444.png,Viral Pneumonia/masks/Viral Pneumonia-444.png,True,G20329,val,233480886a4ce47b45af6f5f9eb41bb0,9435c31ade4abcc6
Viral Pneumonia-445.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-445.png,Viral Pneumonia/masks/Viral Pneumonia-445.png,True,G20330,test,f9d818f2e55e1e14e91f85993ac1ec9e,8019e5c33fa5c62f
Viral Pneumonia-446.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-446.png,Viral Pneumonia/masks/Viral Pneumonia-446.png,True,G20331,train,9eb31716f5cc684ff56fa0010647bf2b,91abc20bc8693deb
Viral Pneumonia-447.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-447.png,Viral Pneumonia/masks/Viral Pneumonia-447.png,True,G20332,test,e6af35cf019dc92e3af8e22284e668f9,91a9c23eca3bd44b
Viral Pneumonia-448.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-448.png,Viral Pneumonia/masks/Viral Pneumonia-448.png,True,G20333,train,d0d50d6bfbc63687c42a68f2ea05603d,907dc71eb0cf3885
Viral Pneumonia-449.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-449.png,Viral Pneumonia/masks/Viral Pneumonia-449.png,True,G20334,train,b61c95bf4a525b08c99698e0958aa0b2,9125c71ade4b3cca
Viral Pneumonia-45.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-45.png,Viral Pneumonia/masks/Viral Pneumonia-45.png,True,G20335,train,0c8bff80617a9713d32b719bf417a1ba,d06eaf1b071a433e
Viral Pneumonia-450.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-450.png,Viral Pneumonia/masks/Viral Pneumonia-450.png,True,G20336,train,f0bfaa3df47f7ba997d051b6e99a82a5,811fe21fecc83ce8
Viral Pneumonia-451.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-451.png,Viral Pneumonia/masks/Viral Pneumonia-451.png,True,G20337,train,08984e64797ad39847ee2427e4a2888b,d06fc71b19163995
Viral Pneumonia-452.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-452.png,Viral Pneumonia/masks/Viral Pneumonia-452.png,True,G20338,train,0c0f07a287d2b28b31505c4edd94acd8,802bea5ed8597d49
Viral Pneumonia-453.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-453.png,Viral Pneumonia/masks/Viral Pneumonia-453.png,True,G20339,train,145ce6466c4efa121d7a75b07793d829,d17eb73a8d1b1904
Viral Pneumonia-454.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-454.png,Viral Pneumonia/masks/Viral Pneumonia-454.png,True,G20340,train,9dda4cce0003029512826d6ec29dc507,94f4e11fe8a5a668
Viral Pneumonia-455.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-455.png,Viral Pneumonia/masks/Viral Pneumonia-455.png,True,G20341,train,2a442ee05af2b9766593fa7c3997a0ee,916ac739b8ed28b4
Viral Pneumonia-456.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-456.png,Viral Pneumonia/masks/Viral Pneumonia-456.png,True,G20342,train,c43313fc1c6c4a65db04128479bfc5f2,c19767c69e38c24d
Viral Pneumonia-457.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-457.png,Viral Pneumonia/masks/Viral Pneumonia-457.png,True,G20343,val,83aab89689aa248f9b21f865e01552c4,d072c31bc039f46f
Viral Pneumonia-458.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-458.png,Viral Pneumonia/masks/Viral Pneumonia-458.png,True,G20344,val,cfb94ae031617408513542c9e0b96cab,812bc4463f944bbf
Viral Pneumonia-459.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-459.png,Viral Pneumonia/masks/Viral Pneumonia-459.png,True,G20345,test,de20c14fdb5f08051286af9ee0f2d0be,d47cf40afbc54aa0
Viral Pneumonia-46.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-46.png,Viral Pneumonia/masks/Viral Pneumonia-46.png,True,G20346,test,c3bd96cc4f758f389df739de71c55702,81ebeeface184509
Viral Pneumonia-460.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-460.png,Viral Pneumonia/masks/Viral Pneumonia-460.png,True,G20347,train,3c04c83581f6a64ba5bdf7432f9a79a6,95c4871b8d5ba556
Viral Pneumonia-461.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-461.png,Viral Pneumonia/masks/Viral Pneumonia-461.png,True,G20348,train,d6a4c9914872d8624f86d309635838a7,9145c61e8f5b8d4b
Viral Pneumonia-462.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-462.png,Viral Pneumonia/masks/Viral Pneumonia-462.png,True,G20349,test,b526483bff65bbe8ae6cdc301c329dad,d4f4c239934b91cb
Viral Pneumonia-463.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-463.png,Viral Pneumonia/masks/Viral Pneumonia-463.png,True,G20350,train,1b34a9f4c029d4197873aa6ff416e44d,d0bde21a9e4a3c6c
Viral Pneumonia-464.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-464.png,Viral Pneumonia/masks/Viral Pneumonia-464.png,True,G20351,val,d0c4613ddcdfe40bcbdac796ff4f9469,9475e75ea31b8807
Viral Pneumonia-465.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-465.png,Viral Pneumonia/masks/Viral Pneumonia-465.png,True,G20352,train,9c3629ffdb5fa4282625e812188635fb,d460825bf11fb11f
Viral Pneumonia-466.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-466.png,Viral Pneumonia/masks/Viral Pneumonia-466.png,True,G20353,train,6e1e1e696d9b7faa798bfd1408539857,d47ea71f8d1d2884
Viral Pneumonia-467.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-467.png,Viral Pneumonia/masks/Viral Pneumonia-467.png,True,G20354,train,41b6a41d3886c62a775ff05653c51b70,d0ade71f43455c45
Viral Pneumonia-468.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-468.png,Viral Pneumonia/masks/Viral Pneumonia-468.png,True,G20355,train,781b771aa7d4a31829f2f4d456724946,90abe3bfc45d5441
Viral Pneumonia-469.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-469.png,Viral Pneumonia/masks/Viral Pneumonia-469.png,True,G20356,train,d64b3482bc0fd659600d08db6113ea10,d4aae75f57411c30
Viral Pneumonia-47.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-47.png,Viral Pneumonia/masks/Viral Pneumonia-47.png,True,G20357,train,1b5b48c079e8fb17193b8ad4b2630bb9,91ffae0a061e1d1e
Viral Pneumonia-470.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-470.png,Viral Pneumonia/masks/Viral Pneumonia-470.png,True,G20358,train,cfb824bd19e12fb359a34a40ed589d97,95fa821a135d5c1f
Viral Pneumonia-471.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-471.png,Viral Pneumonia/masks/Viral Pneumonia-471.png,True,G20359,train,4414cd46dbe4c5a623c0052289f362a7,d0afe71f41433c46
Viral Pneumonia-472.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-472.png,Viral Pneumonia/masks/Viral Pneumonia-472.png,True,G20360,train,7b2ac8b97d9623022dab25740c86d9a9,946ec71bcb0739c1
Viral Pneumonia-473.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-473.png,Viral Pneumonia/masks/Viral Pneumonia-473.png,True,G20361,train,201e8fcce6d219726e0d421344b9a972,803be7c77a3cc4c2
Viral Pneumonia-474.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-474.png,Viral Pneumonia/masks/Viral Pneumonia-474.png,True,G20362,train,9d887f552972bcd825e32ee4015c1789,d474f71fec0c2c0c
Viral Pneumonia-475.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-475.png,Viral Pneumonia/masks/Viral Pneumonia-475.png,True,G20363,train,fd45190fc4732f6e9f9c59f5a455c6b9,d0edc73bed38280c
Viral Pneumonia-476.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-476.png,Viral Pneumonia/masks/Viral Pneumonia-476.png,True,G20364,train,587541e556c6f91e8cea1c6eca964ead,91bbf25fd44a3c40
Viral Pneumonia-477.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-477.png,Viral Pneumonia/masks/Viral Pneumonia-477.png,True,G20365,val,3c28f31c4cddf48d66c6e1af848b1275,d062d65ae57da506
Viral Pneumonia-478.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-478.png,Viral Pneumonia/masks/Viral Pneumonia-478.png,True,G20366,train,4b95501b1dc279498b07b1fab52bd6d7,d02beb7b687c6828
Viral Pneumonia-479.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-479.png,Viral Pneumonia/masks/Viral Pneumonia-479.png,True,G20367,val,88068d4865267077f47e188097777816,d025e39eea1a2ba5
Viral Pneumonia-48.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-48.png,Viral Pneumonia/masks/Viral Pneumonia-48.png,True,G20368,val,e0597531826ccf298d00c6c817a0f2eb,d47cc74fc5546c44
Viral Pneumonia-480.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-480.png,Viral Pneumonia/masks/Viral Pneumonia-480.png,True,G20369,test,66903491d3c34fe598a8e5547325c69f,d037c35fec4d9841
Viral Pneumonia-481.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-481.png,Viral Pneumonia/masks/Viral Pneumonia-481.png,True,G20370,test,858ceaabeff054337fc748f9d97cd2e4,d46de33fe92ca808
Viral Pneumonia-482.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-482.png,Viral Pneumonia/masks/Viral Pneumonia-482.png,True,G20371,train,bb1bb5aced22efa788d81651732a7eec,d4faa32fa52a28a6
Viral Pneumonia-483.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-483.png,Viral Pneumonia/masks/Viral Pneumonia-483.png,True,G20372,test,29e24c79daffaaf95a71801eb9dfba5a,807fef5bc90d4a41
Viral Pneumonia-484.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-484.png,Viral Pneumonia/masks/Viral Pneumonia-484.png,True,G20373,train,00bb4a377e827b37481b9a4092e4913f,91eeef5b4518414d
Viral Pneumonia-485.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-485.png,Viral Pneumonia/masks/Viral Pneumonia-485.png,True,G20374,train,821a134b3adfb347528604448f38bcd3,d125e21eee693cc1
Viral Pneumonia-486.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-486.png,Viral Pneumonia/masks/Viral Pneumonia-486.png,True,G20375,train,80fa0fe0387d75498d6ee4659a60feac,d474875be7398116
Viral Pneumonia-487.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-487.png,Viral Pneumonia/masks/Viral Pneumonia-487.png,True,G20376,train,2e71cff7f4b950bc822aea94f826016e,d025c33fe93d6825
Viral Pneumonia-488.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-488.png,Viral Pneumonia/masks/Viral Pneumonia-488.png,True,G20377,train,9d163697aff958278e72c29700405c27,812ae63eda5f1643
Viral Pneumonia-489.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-489.png,Viral Pneumonia/masks/Viral Pneumonia-489.png,True,G20378,train,5ac95d15ae147a1ef53b1c2fecb261a2,9535c30a904bfec7
Viral Pneumonia-49.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-49.png,Viral Pneumonia/masks/Viral Pneumonia-49.png,True,G20379,train,1c61318348fa5fe996e5b5d9e368d150,d929e25afa4a7849
Viral Pneumonia-490.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-490.png,Viral Pneumonia/masks/Viral Pneumonia-490.png,True,G20380,train,8634571cd1c219c3b88b9b53af9ed941,d07ac63ebc253ca4
Viral Pneumonia-491.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-491.png,Viral Pneumonia/masks/Viral Pneumonia-491.png,True,G20381,train,d8421ff8161daa38b45f8df046c26537,c066f33da53d2c2c
Viral Pneumonia-492.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-492.png,Viral Pneumonia/masks/Viral Pneumonia-492.png,True,G20382,val,c47e66f4047095d2049b95de431956f0,956cb6388d3d8c3c
Viral Pneumonia-493.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-493.png,Viral Pneumonia/masks/Viral Pneumonia-493.png,True,G20383,val,e57c4ae491281f8df6bacfb393d50cfa,803dc63ebac23771
Viral Pneumonia-494.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-494.png,Viral Pneumonia/masks/Viral Pneumonia-494.png,True,G20384,val,a81343d63cacd91e8740e371aab06420,c0e2c75bc34f694a
Viral Pneumonia-495.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-495.png,Viral Pneumonia/masks/Viral Pneumonia-495.png,True,G20385,train,42b5d5cae7f64ac7f2142b14eba93361,947cc15e3a855ef0
Viral Pneumonia-496.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-496.png,Viral Pneumonia/masks/Viral Pneumonia-496.png,True,G20386,test,8b97276827a76a4fdd209adc94f4e1e5,d01ac30b9ccd3df4
Viral Pneumonia-497.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-497.png,Viral Pneumonia/masks/Viral Pneumonia-497.png,True,G20387,train,efac0ea86584814ab2b9b6f05798f68f,d025c31fc54db4e3
Viral Pneumonia-498.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-498.png,Viral Pneumonia/masks/Viral Pneumonia-498.png,True,G20388,train,87c7eb0c76ccec01276e3ad4140d563f,9107c61ae969bd4b
Viral Pneumonia-499.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-499.png,Viral Pneumonia/masks/Viral Pneumonia-499.png,True,G20389,train,9ec9852d3b2a168b4ef0b60a53fd4585,90fec73c8d39844b
Viral Pneumonia-5.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-5.png,Viral Pneumonia/masks/Viral Pneumonia-5.png,True,G20390,val,e5e5f81cef108b5d395e9a01168388fc,d066c65ae93de82c
Viral Pneumonia-50.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-50.png,Viral Pneumonia/masks/Viral Pneumonia-50.png,True,G20391,train,67b7c2b1135c4b156e5ad1ad2592a09a,d105c71ae56bbc61
Viral Pneumonia-500.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-500.png,Viral Pneumonia/masks/Viral Pneumonia-500.png,True,G20392,train,aca74901f7d349a7a05f7ba77917685d,c03fd75b9338c238
Viral Pneumonia-501.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-501.png,Viral Pneumonia/masks/Viral Pneumonia-501.png,True,G20393,train,4dd07af66771c23ed03855db6c506302,917ee778ad78204c
Viral Pneumonia-502.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-502.png,Viral Pneumonia/masks/Viral Pneumonia-502.png,True,G20394,test,2c01f29f032d73a290f8285048671d54,80d5c71aea8f38e5
Viral Pneumonia-503.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-503.png,Viral Pneumonia/masks/Viral Pneumonia-503.png,True,G20395,test,191c5bb3096165f4207cc35cedbbe8dd,d42fe73ba30b3860
Viral Pneumonia-504.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-504.png,Viral Pneumonia/masks/Viral Pneumonia-504.png,True,G20396,train,c92a34ea212d1d3de2b419f8777a8fb7,d4e6f79f85198128
Viral Pneumonia-505.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-505.png,Viral Pneumonia/masks/Viral Pneumonia-505.png,True,G20397,train,f96453c21435f05e83974819ab8968e2,d076d73b850d8587
Viral Pneumonia-506.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-506.png,Viral Pneumonia/masks/Viral Pneumonia-506.png,True,G20398,train,ea53c22fb3c05d07b38f185851a7921a,c12be24b7a687a78
Viral Pneumonia-507.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-507.png,Viral Pneumonia/masks/Viral Pneumonia-507.png,True,G20399,train,5d1e3cd9a4e35802814dd7c33abee540,d476965b8319833d
Viral Pneumonia-508.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-508.png,Viral Pneumonia/masks/Viral Pneumonia-508.png,True,G20400,train,1a965c579469e997fbd8c5e5ed3c6573,c03de34b1ad34f1c
Viral Pneumonia-509.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-509.png,Viral Pneumonia/masks/Viral Pneumonia-509.png,True,G20401,train,93e4b1ba659c1266390111e90068bae8,953fb4381d073987
Viral Pneumonia-51.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-51.png,Viral Pneumonia/masks/Viral Pneumonia-51.png,True,G20402,train,5b71a9b75232e413d80476876293b988,d57fab0f8b09c086
Viral Pneumonia-510.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-510.png,Viral Pneumonia/masks/Viral Pneumonia-510.png,True,G20403,train,7766c27c03549bc592469dbc78dc281d,85e6075b9739691c
Viral Pneumonia-511.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-511.png,Viral Pneumonia/masks/Viral Pneumonia-511.png,True,G20404,train,f87ebfc314da21e3d4825b311bb76b39,d464c31be56fb062
Viral Pneumonia-512.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-512.png,Viral Pneumonia/masks/Viral Pneumonia-512.png,True,G20405,train,79b8ee1c046e5bfd064cd618b8a25839,9129e70ebeca38f0
Viral Pneumonia-513.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-513.png,Viral Pneumonia/masks/Viral Pneumonia-513.png,True,G20406,train,8a22189817349f1db3da3ea3cf03a53b,d476d25fe53da001
Viral Pneumonia-514.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-514.png,Viral Pneumonia/masks/Viral Pneumonia-514.png,True,G20407,train,c27c8ca8569862bcce5bfaf72cac9cbf,d47fa75f8d1d0804
Viral Pneumonia-515.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-515.png,Viral Pneumonia/masks/Viral Pneumonia-515.png,True,G20408,train,127aedd87386090ff02cea1532e72496,902ae35ffe4e0ec0
Viral Pneumonia-516.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-516.png,Viral Pneumonia/masks/Viral Pneumonia-516.png,True,G20409,train,43ded3066801cb77a7a8f8e82efccbad,d4fee31fa1091aa1
Viral Pneumonia-517.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-517.png,Viral Pneumonia/masks/Viral Pneumonia-517.png,True,G20410,val,d08326d340bbb15db4d104aed5526e32,d07ec71b8d0d3c94
Viral Pneumonia-518.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-518.png,Viral Pneumonia/masks/Viral Pneumonia-518.png,True,G20411,val,164f95121c7cf2407d973417b045530d,d47ee42e3aa40a7a
Viral Pneumonia-519.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-519.png,Viral Pneumonia/masks/Viral Pneumonia-519.png,True,G20412,test,51e67cdb1e21b0dc3c137718995c785a,d139e56a3d94c22b
Viral Pneumonia-52.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-52.png,Viral Pneumonia/masks/Viral Pneumonia-52.png,True,G20413,test,20e14ac6395387a17c5f9e11280a5538,d47ea73f8f1c2880
Viral Pneumonia-520.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-520.png,Viral Pneumonia/masks/Viral Pneumonia-520.png,True,G20414,test,309119b357719019f9314a93b6f0eb61,912bee3cd04bb6c2
Viral Pneumonia-521.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-521.png,Viral Pneumonia/masks/Viral Pneumonia-521.png,True,G20415,train,89a8c63a83d60694467e301e45087a62,d529e20ae968bde8
Viral Pneumonia-522.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-522.png,Viral Pneumonia/masks/Viral Pneumonia-522.png,True,G20416,train,25cdc98f2254bcd6cdbd40fdb0461b28,907de12abec559a1
Viral Pneumonia-523.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-523.png,Viral Pneumonia/masks/Viral Pneumonia-523.png,True,G20417,train,dc45fdf4b494761e84d0267f7b270027,d425c31fc60b9d4b
Viral Pneumonia-524.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-524.png,Viral Pneumonia/masks/Viral Pneumonia-524.png,True,G20418,train,c5e97643781b2a3f0c5fca0771b8f094,d076a73fc51c2986
Viral Pneumonia-525.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-525.png,Viral Pneumonia/masks/Viral Pneumonia-525.png,True,G20419,train,803cc18a711e2dabf29c06b0c9fe508e,d07c961aed3d6885
Viral Pneumonia-526.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-526.png,Viral Pneumonia/masks/Viral Pneumonia-526.png,True,G20420,train,0427f4f7d1646e510e67dc4e5be2eb11,897c861a8d1b79cd
Viral Pneumonia-527.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-527.png,Viral Pneumonia/masks/Viral Pneumonia-527.png,True,G20421,train,1bd22b96df565e716e08bc1a6a530581,917daf3a891d380d
Viral Pneumonia-528.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-528.png,Viral Pneumonia/masks/Viral Pneumonia-528.png,True,G20422,train,99e3b7690ce3953b43e6a3a324b259da,d07fc35ed44a3c42
Viral Pneumonia-529.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-529.png,Viral Pneumonia/masks/Viral Pneumonia-529.png,True,G20423,train,71d1f20553de8d2a6dca3e4fc6480273,d031c71f91cb2dd1
Viral Pneumonia-53.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-53.png,Viral Pneumonia/masks/Viral Pneumonia-53.png,True,G20424,train,6b4d8d1708ada1e954f5cecbc5796e19,91e4b63aed38a538
Viral Pneumonia-530.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-530.png,Viral Pneumonia/masks/Viral Pneumonia-530.png,True,G20425,train,35bf7a0ceab8f4ccca210eca5404a4cf,9429e31eea6b3c64
Viral Pneumonia-531.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-531.png,Viral Pneumonia/masks/Viral Pneumonia-531.png,True,G20426,train,a99d97895af8ccbfdd9adcf2be904519,d06bfa5a5d405d58
Viral Pneumonia-532.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-532.png,Viral Pneumonia/masks/Viral Pneumonia-532.png,True,G20427,val,00eac1dd52daf8c325e7cb6fdcac69e9,91e4875aad78a53c
Viral Pneumonia-533.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-533.png,Viral Pneumonia/masks/Viral Pneumonia-533.png,True,G20428,val,46dabc7dc70b9c71c273647e5a1e5eec,902fc31ee5693cc9
Viral Pneumonia-534.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-534.png,Viral Pneumonia/masks/Viral Pneumonia-534.png,True,G20429,val,845a683d985b1f61461f1b2946da0588,d474d25ba30fb443
Viral Pneumonia-535.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-535.png,Viral Pneumonia/masks/Viral Pneumonia-535.png,True,G20430,test,adeb048c1db171385f25979cea16ca77,913dc63b94cf38c2
Viral Pneumonia-536.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-536.png,Viral Pneumonia/masks/Viral Pneumonia-536.png,True,G20431,test,1797ac82ecc400a0559eaeae3c76af26,912fc63af8693543
Viral Pneumonia-537.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-537.png,Viral Pneumonia/masks/Viral Pneumonia-537.png,True,G20432,test,8e90402076cd8b7a65d84de3876e0db7,d072f33fc53b8128
Viral Pneumonia-538.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-538.png,Viral Pneumonia/masks/Viral Pneumonia-538.png,True,G20433,train,8becca511008505da878f32f4ab118a0,9174c65abd79a06c
Viral Pneumonia-539.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-539.png,Viral Pneumonia/masks/Viral Pneumonia-539.png,True,G20434,train,1ff06fe966fca03293958b069b513c92,d13fb61f8e590c44
Viral Pneumonia-54.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-54.png,Viral Pneumonia/masks/Viral Pneumonia-54.png,True,G20435,train,66dd48ea2a1a8a97f751068205cc0aa4,d52de30baf433c60
Viral Pneumonia-540.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-540.png,Viral Pneumonia/masks/Viral Pneumonia-540.png,True,G20436,train,1ea24af2bf3bf4a9ef15bb95dfb46da3,d42fb27faa592c40
Viral Pneumonia-541.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-541.png,Viral Pneumonia/masks/Viral Pneumonia-541.png,True,G20437,train,e894bb807d18a7ced5c8b0fd5de8dd94,941c68e1633f3eb4
Viral Pneumonia-542.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-542.png,Viral Pneumonia/masks/Viral Pneumonia-542.png,True,G20438,train,17a7588a54e6c6e36006ab53a242a0c0,d421c31fdf283cc3
Viral Pneumonia-543.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-543.png,Viral Pneumonia/masks/Viral Pneumonia-543.png,True,G20439,train,08cda50c0d7a7649637ceec9c22a32d6,91e5c61ac20ff0cf
Viral Pneumonia-544.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-544.png,Viral Pneumonia/masks/Viral Pneumonia-544.png,True,G20440,train,29d927146f254b74aa4432b456542338,91abe23ec24b3d4a
Viral Pneumonia-545.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-545.png,Viral Pneumonia/masks/Viral Pneumonia-545.png,True,G20441,train,f24170ed96727bf6dcb7aeb5402eba44,9125c31ae03afe6b
Viral Pneumonia-546.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-546.png,Viral Pneumonia/masks/Viral Pneumonia-546.png,True,G20442,test,992482115bb824dbaad46019e7ab0336,d13de10a2be9383d
Viral Pneumonia-547.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-547.png,Viral Pneumonia/masks/Viral Pneumonia-547.png,True,G20443,train,0a4ffde0f8437a42a5932a666b909f5f,d54a5f5e1a595909
Viral Pneumonia-548.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-548.png,Viral Pneumonia/masks/Viral Pneumonia-548.png,True,G20444,val,8332aa91e47bc8d15047556dd2c1a6bd,957ea73a8d382c2c
Viral Pneumonia-549.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-549.png,Viral Pneumonia/masks/Viral Pneumonia-549.png,True,G20445,val,f5e6df474a153dca3880f0eb72d1113b,91abc20ac06b7deb
Viral Pneumonia-55.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-55.png,Viral Pneumonia/masks/Viral Pneumonia-55.png,True,G20446,train,0271798419deb23b47d5243e1e44c572,d4fce71bc71a9202
Viral Pneumonia-550.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-550.png,Viral Pneumonia/masks/Viral Pneumonia-550.png,True,G20447,val,8432be79c284041acca5debf417afca9,91a5c31ec06df44f
Viral Pneumonia-551.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-551.png,Viral Pneumonia/masks/Viral Pneumonia-551.png,True,G20448,train,67ece0f7b997d586a2708dcf7850edc5,d43de0c014fee24f
Viral Pneumonia-552.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-552.png,Viral Pneumonia/masks/Viral Pneumonia-552.png,True,G20449,train,f50a326d3b484f2c8f59efd1357a8c42,916da73aaa39682d
Viral Pneumonia-553.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-553.png,Viral Pneumonia/masks/Viral Pneumonia-553.png,True,G20450,train,a7655da2d642629fc387fb100151936b,9065c71ee938e53a
Viral Pneumonia-554.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-554.png,Viral Pneumonia/masks/Viral Pneumonia-554.png,True,G20451,train,1541e18cafb0558bade41f097aaee1bf,d03cc20b9bc86dfc
Viral Pneumonia-555.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-555.png,Viral Pneumonia/masks/Viral Pneumonia-555.png,True,G20452,test,b43618f70aec122ed255c7bea6035293,d5a8c20ac31af74f
Viral Pneumonia-556.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-556.png,Viral Pneumonia/masks/Viral Pneumonia-556.png,True,G20453,test,92b504abfaeeb7d34e950b9a33515f9d,902fe71e944b3dc2
Viral Pneumonia-557.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-557.png,Viral Pneumonia/masks/Viral Pneumonia-557.png,True,G20454,test,f98b708547a7f31510594b59e998da57,91e5c71ee838e439
Viral Pneumonia-558.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-558.png,Viral Pneumonia/masks/Viral Pneumonia-558.png,True,G20455,train,232711dd0e0059ff3d7e482983a4e053,d066b75a8739893c
Viral Pneumonia-559.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-559.png,Viral Pneumonia/masks/Viral Pneumonia-559.png,True,G20456,train,9aaf8694b0640758797147ae8c9100c3,95aae21ed4491fe8
Viral Pneumonia-56.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-56.png,Viral Pneumonia/masks/Viral Pneumonia-56.png,True,G20457,train,761a9ad7fe02bbe54b83382d44ec0323,80ade35d5cd25e70
Viral Pneumonia-560.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-560.png,Viral Pneumonia/masks/Viral Pneumonia-560.png,True,G20458,train,1bf76e1db5fb29bc0e58a798ade90d34,91ed8f12c71b8738
Viral Pneumonia-561.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-561.png,Viral Pneumonia/masks/Viral Pneumonia-561.png,True,G20459,train,4c2160447648d4cd66118b854921b2b0,d03ec35bd5c93d80
Viral Pneumonia-562.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-562.png,Viral Pneumonia/masks/Viral Pneumonia-562.png,True,G20460,train,085b94b8ad7a23f757292bb636b4a1f8,c06ac75bd18f2d86
Viral Pneumonia-563.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-563.png,Viral Pneumonia/masks/Viral Pneumonia-563.png,True,G20461,train,8b93273d0e4f1558fd13d4d0a7efa956,9042c75f56505ef3
Viral Pneumonia-564.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-564.png,Viral Pneumonia/masks/Viral Pneumonia-564.png,True,G20462,train,c9500af62448b0ec8461e9848c7b0639,913bc65c18595b3d
Viral Pneumonia-565.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-565.png,Viral Pneumonia/masks/Viral Pneumonia-565.png,True,G20463,train,ef1579ab0df05945d1715b39db3ead84,d47ec71bad8d2908
Viral Pneumonia-566.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-566.png,Viral Pneumonia/masks/Viral Pneumonia-566.png,True,G20464,train,13c31bc8a68e7c6e3a4ee597dc90e55c,9029e11ee65abeca
Viral Pneumonia-567.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-567.png,Viral Pneumonia/masks/Viral Pneumonia-567.png,True,G20465,train,132aaaaf3fa52da91b1702572fa90d39,91c7865ac63ac39e
Viral Pneumonia-568.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-568.png,Viral Pneumonia/masks/Viral Pneumonia-568.png,True,G20466,train,f2c6505648ea155b23ea057d89ad037a,d0f4e61bc529b469
Viral Pneumonia-569.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-569.png,Viral Pneumonia/masks/Viral Pneumonia-569.png,True,G20467,val,2d42aed489c81bb7d356563e695b6fbd,812bc61e9c413dfd
Viral Pneumonia-57.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-57.png,Viral Pneumonia/masks/Viral Pneumonia-57.png,True,G20468,train,333ea57260998ac01100d774c7c0ae37,d07ea71bbd0b3920
Viral Pneumonia-570.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-570.png,Viral Pneumonia/masks/Viral Pneumonia-570.png,True,G20469,val,6cae32a120cab35d7cba576e2a7cf97f,d47c971fad1c281c
Viral Pneumonia-571.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-571.png,Viral Pneumonia/masks/Viral Pneumonia-571.png,True,G20470,train,01201797c440132b21afdd458cbbc961,d07ae71ff51d2804
Viral Pneumonia-572.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-572.png,Viral Pneumonia/masks/Viral Pneumonia-572.png,True,G20471,val,27da8dc3e55d688c7cfb7d2dfa836bbe,d03de32fae4b3c40
Viral Pneumonia-573.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-573.png,Viral Pneumonia/masks/Viral Pneumonia-573.png,True,G20472,test,2e0604eaaae27c5ff95da2ffefdec37e,812de20ef86b3d69
Viral Pneumonia-574.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-574.png,Viral Pneumonia/masks/Viral Pneumonia-574.png,True,G20473,train,c4b6984dda26e291ff10a3009ecf24bc,91a7c318e438f46f
Viral Pneumonia-575.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-575.png,Viral Pneumonia/masks/Viral Pneumonia-575.png,True,G20474,train,b6fe0557503aeeeae9dd8311afbc472c,d4ffa30aa2471993
Viral Pneumonia-576.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-576.png,Viral Pneumonia/masks/Viral Pneumonia-576.png,True,G20475,test,ac1a22b9ed144cdd2a44c5d1339aeab5,d02be61e1d411d5f
Viral Pneumonia-577.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-577.png,Viral Pneumonia/masks/Viral Pneumonia-577.png,True,G20476,train,196d83b73fd32978b6543a0c77aa4a25,811fc61eb04abdd3
Viral Pneumonia-578.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-578.png,Viral Pneumonia/masks/Viral Pneumonia-578.png,True,G20477,train,25e16b64cdc8ef7dab0174e2ce0b9e1c,97afa809555f2aa2
Viral Pneumonia-579.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-579.png,Viral Pneumonia/masks/Viral Pneumonia-579.png,True,G20478,test,6cfb6779de86d5241d224c5edd2be90a,c02fea4fde491d48
Viral Pneumonia-58.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-58.png,Viral Pneumonia/masks/Viral Pneumonia-58.png,True,G20479,train,6dd9e1f3ee849fe2a49adab5cea81782,d42b6b2f7c4e4c18
Viral Pneumonia-580.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-580.png,Viral Pneumonia/masks/Viral Pneumonia-580.png,True,G20480,train,7d19405c852bb0e3a95d8ad13529406c,9074c71fd34f904b
Viral Pneumonia-581.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-581.png,Viral Pneumonia/masks/Viral Pneumonia-581.png,True,G20481,test,5482a00a29355d943d1a096409cb01c7,905eaaa72bc5075d
Viral Pneumonia-582.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-582.png,Viral Pneumonia/masks/Viral Pneumonia-582.png,True,G20482,train,aa2b320d1d245f81900b3295006b2edc,9165e33fe05ba468
Viral Pneumonia-583.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-583.png,Viral Pneumonia/masks/Viral Pneumonia-583.png,True,G20483,train,bafedb63b8d2610297031de5e408ede9,c4b4e21bb74bb642
Viral Pneumonia-584.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-584.png,Viral Pneumonia/masks/Viral Pneumonia-584.png,True,G20484,train,ddcbf873149f4fb56550109b87fe6f65,80b5e55bb0c66f34
Viral Pneumonia-585.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-585.png,Viral Pneumonia/masks/Viral Pneumonia-585.png,True,G20485,train,c7d19dab8c51c68169321767d652251c,c439e1473a7416be
Viral Pneumonia-586.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-586.png,Viral Pneumonia/masks/Viral Pneumonia-586.png,True,G20486,val,f2b17c6101a902e8e21575b6e41c5240,d45ce20ff0873b34
Viral Pneumonia-587.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-587.png,Viral Pneumonia/masks/Viral Pneumonia-587.png,True,G20487,train,370b67776f9ee4ecb01e9be5ebb74f6b,8001eb97fcc26ef1
Viral Pneumonia-588.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-588.png,Viral Pneumonia/masks/Viral Pneumonia-588.png,True,G20488,train,a0ae775b622f03a00f9f7c5f0b1e800e,81b5c61ae038f56f
Viral Pneumonia-589.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-589.png,Viral Pneumonia/masks/Viral Pneumonia-589.png,True,G20489,test,4b29a2d7380a64776d07849a73ae719d,d02fc35f5e4b1c48
Viral Pneumonia-59.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-59.png,Viral Pneumonia/masks/Viral Pneumonia-59.png,True,G20490,train,119e14d9e376223851ec0bb1ab53e507,902bea3ada197c4b
Viral Pneumonia-590.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-590.png,Viral Pneumonia/masks/Viral Pneumonia-590.png,True,G20491,train,2f5bf845d1a8e49cf9e16cd8e133c0c3,c06fd71f9d0f1884
Viral Pneumonia-591.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-591.png,Viral Pneumonia/masks/Viral Pneumonia-591.png,True,G20492,train,cd1a9fca25d625179d41ca0887c60449,d4b4e31ba14b3c63
Viral Pneumonia-592.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-592.png,Viral Pneumonia/masks/Viral Pneumonia-592.png,True,G20493,val,79f6712256fe8a71a42c08351b77bb71,d12bfa38342935b6
Viral Pneumonia-593.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-593.png,Viral Pneumonia/masks/Viral Pneumonia-593.png,True,G20494,val,f9911819293ffb862e285ef2bebe29c4,c0feaf0dc04d9547
Viral Pneumonia-594.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-594.png,Viral Pneumonia/masks/Viral Pneumonia-594.png,True,G20495,train,dfb31039c18e1610a3ef2bc70619675c,d02fb5555f4e0a2c
Viral Pneumonia-595.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-595.png,Viral Pneumonia/masks/Viral Pneumonia-595.png,True,G20496,train,770a1ed23df623f771fdecc5c06b464f,812ff6a033abe296
Viral Pneumonia-596.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-596.png,Viral Pneumonia/masks/Viral Pneumonia-596.png,True,G20496,train,770a1ed23df623f771fdecc5c06b464f,812ff6a033abe296
Viral Pneumonia-597.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-597.png,Viral Pneumonia/masks/Viral Pneumonia-597.png,True,G20497,train,0210e69328e25a002eba27e9081588a8,913966ce7c5c4a2d
Viral Pneumonia-598.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-598.png,Viral Pneumonia/masks/Viral Pneumonia-598.png,True,G20498,train,969cebebee932de25e083c4c31401cbe,802edf1e565a3d49
Viral Pneumonia-599.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-599.png,Viral Pneumonia/masks/Viral Pneumonia-599.png,True,G20499,train,8599c01f40bd86439695a0ebf55676d4,c02f971b5f4b6862
Viral Pneumonia-6.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-6.png,Viral Pneumonia/masks/Viral Pneumonia-6.png,True,G20500,train,f9f020577043752933b622c68c2aa48a,9135c73ec20b3ac7
Viral Pneumonia-60.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-60.png,Viral Pneumonia/masks/Viral Pneumonia-60.png,True,G20501,val,9ed10473ff5424eb6a31b7d1a805acc0,d0fda71aab3ba006
Viral Pneumonia-600.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-600.png,Viral Pneumonia/masks/Viral Pneumonia-600.png,True,G20502,train,2709e39b0dbfaf5b271a2a1c7e431297,952bc33ec23ab06e
Viral Pneumonia-601.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-601.png,Viral Pneumonia/masks/Viral Pneumonia-601.png,True,G20503,train,0d0256131f3b726c516d8409285e1e09,9129c71aca6a786f
Viral Pneumonia-602.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-602.png,Viral Pneumonia/masks/Viral Pneumonia-602.png,True,G20504,train,76864ede30a0a30b25a32a88eaba2466,91a5c61acb6b3c69
Viral Pneumonia-603.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-603.png,Viral Pneumonia/masks/Viral Pneumonia-603.png,True,G20505,train,835977cccb07c2e23acb5ddea58ce2c1,9469ef9ec318c31a
Viral Pneumonia-604.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-604.png,Viral Pneumonia/masks/Viral Pneumonia-604.png,True,G20506,train,aebd9cab4c7c558bc50eb463756a2df5,9468e31fc31bd94a
Viral Pneumonia-605.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-605.png,Viral Pneumonia/masks/Viral Pneumonia-605.png,True,G20507,train,c2d608467d08bcfe675f88e6e62bec72,9429e31fd6c61e52
Viral Pneumonia-606.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-606.png,Viral Pneumonia/masks/Viral Pneumonia-606.png,True,G20508,train,7e8ec38d32ca3ee3871baafa515f56de,812fc64b98cb3bb2
Viral Pneumonia-607.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-607.png,Viral Pneumonia/masks/Viral Pneumonia-607.png,True,G20509,val,18b64f2ca79a699cc9ba49de3757d874,913cc63839e9799c
Viral Pneumonia-608.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-608.png,Viral Pneumonia/masks/Viral Pneumonia-608.png,True,G20510,val,7c4c24abac5e40d583b1e125167e44a2,d03ac71c961db41f
Viral Pneumonia-609.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-609.png,Viral Pneumonia/masks/Viral Pneumonia-609.png,True,G20511,test,dc6d82395587a1f5bb022a51fa90d332,913fd61cc11db887
Viral Pneumonia-61.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-61.png,Viral Pneumonia/masks/Viral Pneumonia-61.png,True,G20512,test,5e3b58aa92fc83aef6a67150c0c5121a,d124c21fed3ee461
Viral Pneumonia-610.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-610.png,Viral Pneumonia/masks/Viral Pneumonia-610.png,True,G20513,test,caa66164337a7ec6c89e1b5e91e8fafe,911965c36f9cf6c0
Viral Pneumonia-611.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-611.png,Viral Pneumonia/masks/Viral Pneumonia-611.png,True,G20514,train,357401589a193610ec23cd659d75427b,dc6e871b913b3845
Viral Pneumonia-612.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-612.png,Viral Pneumonia/masks/Viral Pneumonia-612.png,True,G20515,train,7b000dfa5b8fed6f3e40bdfa1f4ca5f9,9038e1473ee44bb7
Viral Pneumonia-613.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-613.png,Viral Pneumonia/masks/Viral Pneumonia-613.png,True,G20516,train,4929ece3e65cd47e50efd139897d0b32,9165a77ae40e6987
Viral Pneumonia-614.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-614.png,Viral Pneumonia/masks/Viral Pneumonia-614.png,True,G20517,train,dc6dd1d09041bc16bc1ccbfbee3f0bc5,d07be70e831b3d81
Viral Pneumonia-615.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-615.png,Viral Pneumonia/masks/Viral Pneumonia-615.png,True,G20518,train,00c4ca3ea87add7d09677e2a14e9e059,903b561e924f94af
Viral Pneumonia-616.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-616.png,Viral Pneumonia/masks/Viral Pneumonia-616.png,True,G20519,train,b8a6f0d7b157a69fe9bf72a186afde2a,9474c61bc34bb16b
Viral Pneumonia-617.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-617.png,Viral Pneumonia/masks/Viral Pneumonia-617.png,True,G20520,train,1019856e97299c5657d2c75d6054cc6a,9568e63a9e0e3987
Viral Pneumonia-618.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-618.png,Viral Pneumonia/masks/Viral Pneumonia-618.png,True,G20521,train,90d1af8f4df0d21daab17226c7ba80b0,c47ea32fc1974991
Viral Pneumonia-619.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-619.png,Viral Pneumonia/masks/Viral Pneumonia-619.png,True,G20522,train,31d214cabad0dbee907c1d9a13670328,906aebbec63a921a
Viral Pneumonia-62.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-62.png,Viral Pneumonia/masks/Viral Pneumonia-62.png,True,G20523,train,58eb05f908914169850509c0e9e098cc,90e5c71ae51ee44e
Viral Pneumonia-620.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-620.png,Viral Pneumonia/masks/Viral Pneumonia-620.png,True,G20524,train,df0200634da87966eeb0c540091dc1e5,d47cc71b831b2996
Viral Pneumonia-621.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-621.png,Viral Pneumonia/masks/Viral Pneumonia-621.png,True,G20525,val,22d9b129dba6684181f844fb8e31d30f,d43f8f2d0d2c085f
Viral Pneumonia-622.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-622.png,Viral Pneumonia/masks/Viral Pneumonia-622.png,True,G20526,val,19d8eb6c481718d3be95d7b737ecde9f,c125c21be06b7c6f
Viral Pneumonia-623.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-623.png,Viral Pneumonia/masks/Viral Pneumonia-623.png,True,G20527,test,a4767f46d31a23816984e42b7e8564c4,91ee06138f79ada4
Viral Pneumonia-624.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-624.png,Viral Pneumonia/masks/Viral Pneumonia-624.png,True,G20528,train,bfade0c5a162573ac0a059af954859d3,9079e5473ec6ca1a
Viral Pneumonia-625.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-625.png,Viral Pneumonia/masks/Viral Pneumonia-625.png,True,G20529,train,fa06328e34a8e7e741c84da6c9efa57f,917a4f1a194579e9
Viral Pneumonia-626.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-626.png,Viral Pneumonia/masks/Viral Pneumonia-626.png,True,G20530,train,05483d82f3517772a4291a560c77d9ef,9039e5c76e39e542
Viral Pneumonia-627.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-627.png,Viral Pneumonia/masks/Viral Pneumonia-627.png,True,G20531,train,acf9c795e8acaea750cce804708a32ca,912fc61e1c4f3d38
Viral Pneumonia-628.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-628.png,Viral Pneumonia/masks/Viral Pneumonia-628.png,True,G20532,train,935c69f92a555217741c64cf6e3b46d6,80a1e73ef4ca3cf0
Viral Pneumonia-629.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-629.png,Viral Pneumonia/masks/Viral Pneumonia-629.png,True,G20533,val,e64cf6473bbc9dcc8a8844e710595637,d02fa738f16d3063
Viral Pneumonia-63.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-63.png,Viral Pneumonia/masks/Viral Pneumonia-63.png,True,G20534,test,fa61359fbbf39d5277b052d53cbd4ed9,913dc70ff2693069
Viral Pneumonia-630.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-630.png,Viral Pneumonia/masks/Viral Pneumonia-630.png,True,G20535,test,d27f2f8cf10b2ffeb043d862c74b377e,942ae33afe691e60
Viral Pneumonia-631.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-631.png,Viral Pneumonia/masks/Viral Pneumonia-631.png,True,G20536,train,a3e5a58b4c85cfe8c95e6df708b2743f,c06fa31bc54d4dc6
Viral Pneumonia-632.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-632.png,Viral Pneumonia/masks/Viral Pneumonia-632.png,True,G20537,val,919f116c545ecc9c4434e1bbe6d0d03e,c02f770f613c662d
Viral Pneumonia-633.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-633.png,Viral Pneumonia/masks/Viral Pneumonia-633.png,True,G20538,train,b7a4b044d0da5b7d1be1f8686a0a1fd9,d07ea31f0a0d3e3c
Viral Pneumonia-634.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-634.png,Viral Pneumonia/masks/Viral Pneumonia-634.png,True,G20539,train,0d2ab20eefb7755b877b39929fdb2eee,80b6e71f53197c25
Viral Pneumonia-635.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-635.png,Viral Pneumonia/masks/Viral Pneumonia-635.png,True,G20540,train,4d5a993775bd12af7ce514a15e98bfd4,903ba5c31fb4d24e
Viral Pneumonia-636.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-636.png,Viral Pneumonia/masks/Viral Pneumonia-636.png,True,G20541,train,4353339fdef5de6883be2d3638f36ff9,8039e71e34d25ebc
Viral Pneumonia-637.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-637.png,Viral Pneumonia/masks/Viral Pneumonia-637.png,True,G20542,train,449611b7a25b35285f7bb94c16b2b5d7,d02fef1dc30f814a
Viral Pneumonia-638.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-638.png,Viral Pneumonia/masks/Viral Pneumonia-638.png,True,G20543,train,77bf8aaf5f63b19c22d3f481347c366f,913ff2569a0b1c53
Viral Pneumonia-639.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-639.png,Viral Pneumonia/masks/Viral Pneumonia-639.png,True,G20544,train,f7ca4a4c35d3f4598998d2b7a4ec9f62,c82df51cc34b98cb
Viral Pneumonia-64.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-64.png,Viral Pneumonia/masks/Viral Pneumonia-64.png,True,G20545,train,5186f3d91f2153adf39c1b9a78ae24bd,d4fca72b87135514
Viral Pneumonia-640.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-640.png,Viral Pneumonia/masks/Viral Pneumonia-640.png,True,G20546,test,fec50c0fe655c458057eea545bd3bd48,d12f6218d16b3c67
Viral Pneumonia-641.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-641.png,Viral Pneumonia/masks/Viral Pneumonia-641.png,True,G20547,train,0a6e443f4fda0ae3133fc873564d8f33,952ffb1ed61e1840
Viral Pneumonia-642.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-642.png,Viral Pneumonia/masks/Viral Pneumonia-642.png,True,G20548,train,c2f5de109d09a7f682f7de67040f01bd,986be61ec03ff40a
Viral Pneumonia-643.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-643.png,Viral Pneumonia/masks/Viral Pneumonia-643.png,True,G20549,train,0fc25b44bb1d0d4a3199fdd081ebb6ef,952bfe7cbc01093a
Viral Pneumonia-644.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-644.png,Viral Pneumonia/masks/Viral Pneumonia-644.png,True,G20550,val,086864208a48ab9e377a57ff0793f5a2,957e230617531f6c
Viral Pneumonia-645.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-645.png,Viral Pneumonia/masks/Viral Pneumonia-645.png,True,G20551,test,c436ccdc342bbe01fbe9e2d470ec88ea,9479e796d21ae107
Viral Pneumonia-646.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-646.png,Viral Pneumonia/masks/Viral Pneumonia-646.png,True,G20552,val,e483fc3db275dfd79390dfd3f5a13a63,d474a70aa2877aa7
Viral Pneumonia-647.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-647.png,Viral Pneumonia/masks/Viral Pneumonia-647.png,True,G20553,test,6fde6df5e9fd560ddeb036ec57edcda7,c06ea719836de569
Viral Pneumonia-648.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-648.png,Viral Pneumonia/masks/Viral Pneumonia-648.png,True,G20554,train,31fe27c96d0b201bd09c0bbefb3ba296,9139e73ab04ab1c7
Viral Pneumonia-649.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-649.png,Viral Pneumonia/masks/Viral Pneumonia-649.png,True,G20555,train,a299346c764a3af5814f7421d68914f6,df3fe01e545538c0
Viral Pneumonia-65.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-65.png,Viral Pneumonia/masks/Viral Pneumonia-65.png,True,G20556,train,7e99e68a0f8d8abe15332338279ed9d6,d87ce79793971900
Viral Pneumonia-650.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-650.png,Viral Pneumonia/masks/Viral Pneumonia-650.png,True,G20557,train,958780708464a83f290c2d21bab02931,d03f66461c79613d
Viral Pneumonia-651.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-651.png,Viral Pneumonia/masks/Viral Pneumonia-651.png,True,G20558,train,3e18677576128a3260be4529709a4536,d829e14bb7c085fa
Viral Pneumonia-652.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-652.png,Viral Pneumonia/masks/Viral Pneumonia-652.png,True,G20559,train,631bd1f5fd3bf9c4629754b4644513aa,9c7c656eb025c2ba
Viral Pneumonia-653.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-653.png,Viral Pneumonia/masks/Viral Pneumonia-653.png,True,G20560,train,2af72b17319b68ae09f11e63897a9a52,9966c69acf798069
Viral Pneumonia-654.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-654.png,Viral Pneumonia/masks/Viral Pneumonia-654.png,True,G20561,train,6b214b743315dad58470f051d92edd25,9439e10ab41ad3f7
Viral Pneumonia-655.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-655.png,Viral Pneumonia/masks/Viral Pneumonia-655.png,True,G20562,train,6b3a31ff3a367a8bc7438cd955d324c2,dd2ce03ef2433e84
Viral Pneumonia-656.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-656.png,Viral Pneumonia/masks/Viral Pneumonia-656.png,True,G20563,train,7f03c45307d1309fb62018a723972692,df24801bcc39de69
Viral Pneumonia-657.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-657.png,Viral Pneumonia/masks/Viral Pneumonia-657.png,True,G20564,train,537d197b6d86d5dcd623c1958caa14d9,c83f671e2a0f98c3
Viral Pneumonia-658.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-658.png,Viral Pneumonia/masks/Viral Pneumonia-658.png,True,G20565,test,c023c32e72b81eccfdd40596914586f1,913fd67e5a683034
Viral Pneumonia-659.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-659.png,Viral Pneumonia/masks/Viral Pneumonia-659.png,True,G20566,train,fce5333fe0ed16f88563ac0b627cfd0c,856ab75ed11f5811
Viral Pneumonia-66.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-66.png,Viral Pneumonia/masks/Viral Pneumonia-66.png,True,G20567,train,b03139f4a37133cfe0b12c9ca87e61e6,9179c71ed65a344a
Viral Pneumonia-660.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-660.png,Viral Pneumonia/masks/Viral Pneumonia-660.png,True,G20568,train,849513f81fc482053b55c0ad76ba5d1e,d42ad31fd4924f54
Viral Pneumonia-661.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-661.png,Viral Pneumonia/masks/Viral Pneumonia-661.png,True,G20569,train,66a636692e216bc7dc03aea416d934b0,9039521e1e1f3d3d
Viral Pneumonia-662.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-662.png,Viral Pneumonia/masks/Viral Pneumonia-662.png,True,G20570,train,a88159aec4a16b1b620ff4d90eea7477,9032d65dd33db039
Viral Pneumonia-663.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-663.png,Viral Pneumonia/masks/Viral Pneumonia-663.png,True,G20571,val,21b06e069fb97bb5400b3ab38c17dee6,95f36a38c72e814e
Viral Pneumonia-664.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-664.png,Viral Pneumonia/masks/Viral Pneumonia-664.png,True,G20572,val,bba2a2c2068556f7eaca590a8b8d116f,d460693de48fa48f
Viral Pneumonia-665.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-665.png,Viral Pneumonia/masks/Viral Pneumonia-665.png,True,G20573,val,57461ae76b315b26e82c1a178904eecc,d42ad30fc04b7dd2
Viral Pneumonia-666.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-666.png,Viral Pneumonia/masks/Viral Pneumonia-666.png,True,G20574,test,7297bad3dc63f594309b4a79ae61f19d,913bd65a59193917
Viral Pneumonia-667.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-667.png,Viral Pneumonia/masks/Viral Pneumonia-667.png,True,G20575,test,e572a591d8c0c952a4a4b5fd62971364,91bfd61cd24d904b
Viral Pneumonia-668.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-668.png,Viral Pneumonia/masks/Viral Pneumonia-668.png,True,G20576,val,e029ec506f085490053a5542061fade0,d4ec871bad3ccc0c
Viral Pneumonia-669.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-669.png,Viral Pneumonia/masks/Viral Pneumonia-669.png,True,G20577,train,5918b1699641a7fa4490e192d2913298,95fa8203575d154f
Viral Pneumonia-67.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-67.png,Viral Pneumonia/masks/Viral Pneumonia-67.png,True,G20578,train,0c6ecad9ec7ffc1524ea964b8dc2bd50,d42f2d2b17535217
Viral Pneumonia-670.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-670.png,Viral Pneumonia/masks/Viral Pneumonia-670.png,True,G20579,train,6f943b9f133e50b168554f54b4d8a9ce,d46fa70f8d2e4446
Viral Pneumonia-671.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-671.png,Viral Pneumonia/masks/Viral Pneumonia-671.png,True,G20580,train,0a7eb660d003e7319fde73346b0bf279,9460c31bcd6db44f
Viral Pneumonia-672.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-672.png,Viral Pneumonia/masks/Viral Pneumonia-672.png,True,G20581,train,c5cb9a93fcae0fc6a6344d4094cca092,c0fea758d17cc066
Viral Pneumonia-673.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-673.png,Viral Pneumonia/masks/Viral Pneumonia-673.png,True,G20582,train,ebe5d22f45005d75a18378bcc40e7944,916d871a865eb387
Viral Pneumonia-674.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-674.png,Viral Pneumonia/masks/Viral Pneumonia-674.png,True,G20583,train,80ae704a85e5f5392cb70f1e3de8bc13,816f265a417d69e5
Viral Pneumonia-675.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-675.png,Viral Pneumonia/masks/Viral Pneumonia-675.png,True,G20584,train,05731d6427156aa8a7282f77b9e29945,916fde180a78cb4b
Viral Pneumonia-676.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-676.png,Viral Pneumonia/masks/Viral Pneumonia-676.png,True,G20585,train,b9457592ea501599cd3d9aba67d0136a,d47fc71f8b0d08c4
Viral Pneumonia-677.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-677.png,Viral Pneumonia/masks/Viral Pneumonia-677.png,True,G20586,val,3c461c26f1ab441852afeb8454c27abe,907ae79e1a2a296d
Viral Pneumonia-678.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-678.png,Viral Pneumonia/masks/Viral Pneumonia-678.png,True,G20587,train,ef1b470822c8066ec16b9a2fd9e3a461,91a9c31bc34f3c4b
Viral Pneumonia-679.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-679.png,Viral Pneumonia/masks/Viral Pneumonia-679.png,True,G20588,val,dcc6046616c8504642f7018b7c69f551,d0feaf3da50e2086
Viral Pneumonia-68.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-68.png,Viral Pneumonia/masks/Viral Pneumonia-68.png,True,G20589,train,b2b27c14efd95435f8924b4f444d8e33,d0297a535f5e5a28
Viral Pneumonia-680.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-680.png,Viral Pneumonia/masks/Viral Pneumonia-680.png,True,G20590,train,17cf87907499ffd438330983209c270c,907b7a5e5c0a3a2b
Viral Pneumonia-681.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-681.png,Viral Pneumonia/masks/Viral Pneumonia-681.png,True,G20591,test,3993d2a8b9f088798307113d4ba789d6,940ae1b7d7e25a4a
Viral Pneumonia-682.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-682.png,Viral Pneumonia/masks/Viral Pneumonia-682.png,True,G20592,test,fd2a02309daaff6f1aa9d6d55e06fa21,d06bc35f4159585d
Viral Pneumonia-683.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-683.png,Viral Pneumonia/masks/Viral Pneumonia-683.png,True,G20593,test,86417fb3a5c80d4297a07edd50e06956,9169eb3eda5a1858
Viral Pneumonia-684.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-684.png,Viral Pneumonia/masks/Viral Pneumonia-684.png,True,G20594,train,8fc21ce0f8bbc1e1f2ece9a3ef9643b8,dc2ce23fe05b5c44
Viral Pneumonia-685.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-685.png,Viral Pneumonia/masks/Viral Pneumonia-685.png,True,G20595,train,c0898ebe53d958454cdcf883720f8a80,9defa23aa51a811b
Viral Pneumonia-686.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-686.png,Viral Pneumonia/masks/Viral Pneumonia-686.png,True,G20596,train,4858f489e69a00854c8e6994e3de5665,91aae63afc4d384a
Viral Pneumonia-687.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-687.png,Viral Pneumonia/masks/Viral Pneumonia-687.png,True,G20597,train,73f20555ab50abfc02be6050aa8297b5,996e071a075a5b7c
Viral Pneumonia-688.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-688.png,Viral Pneumonia/masks/Viral Pneumonia-688.png,True,G20598,train,3f31b77d859dc4f6f35b5573a86e346a,90f5e716c11e6197
Viral Pneumonia-689.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-689.png,Viral Pneumonia/masks/Viral Pneumonia-689.png,True,G20599,train,bb4b2160d153aaae5821ec26c550c89a,9920c21bc74f6ee5
Viral Pneumonia-69.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-69.png,Viral Pneumonia/masks/Viral Pneumonia-69.png,True,G20600,train,46f95b878595c9aa3a14a43144eedf80,903b72161e5a7d3a
Viral Pneumonia-690.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-690.png,Viral Pneumonia/masks/Viral Pneumonia-690.png,True,G20601,train,90a838c98e5538f7685d36eec14b288d,d06e071a8f5b855e
Viral Pneumonia-691.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-691.png,Viral Pneumonia/masks/Viral Pneumonia-691.png,True,G20602,train,c3f697856c8d9eaefa64c46acbfc6179,9cede54e8318c157
Viral Pneumonia-692.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-692.png,Viral Pneumonia/masks/Viral Pneumonia-692.png,True,G20603,train,b1247f3cd8f204dbe764920958219fc9,dc2fc21fc11cd44b
Viral Pneumonia-693.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-693.png,Viral Pneumonia/masks/Viral Pneumonia-693.png,True,G20604,train,c6b06fcda2a37d99bd091c2538e4230a,8121e75bbec21b96
Viral Pneumonia-694.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-694.png,Viral Pneumonia/masks/Viral Pneumonia-694.png,True,G20605,train,7ce837c09882798543f11f13ddb02d1d,c129c23afc693d69
Viral Pneumonia-695.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-695.png,Viral Pneumonia/masks/Viral Pneumonia-695.png,True,G20606,val,31150ab93054dbc9764c3adbedb8380b,916aaa38217d5d5d
Viral Pneumonia-696.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-696.png,Viral Pneumonia/masks/Viral Pneumonia-696.png,True,G20607,train,80ef8d3feb775e4e3caa0c9b068018c3,d02fbf5755504554
Viral Pneumonia-697.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-697.png,Viral Pneumonia/masks/Viral Pneumonia-697.png,True,G20608,train,19b258a7f5f53432bbdc88553b77d38c,d46b6f5e5b490909
Viral Pneumonia-698.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-698.png,Viral Pneumonia/masks/Viral Pneumonia-698.png,True,G20609,val,d7827433e03fce0835e5f20cda77bcec,d47e471ec35b104e
Viral Pneumonia-699.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-699.png,Viral Pneumonia/masks/Viral Pneumonia-699.png,True,G20610,val,0ed4fd50bfd79090daa40b04665ddb50,d46f830fc54d05ce
Viral Pneumonia-7.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-7.png,Viral Pneumonia/masks/Viral Pneumonia-7.png,True,G20611,test,c9fb59bf00fd9d3070ed616ce2a5b029,9125e21ede4a9c67
Viral Pneumonia-70.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-70.png,Viral Pneumonia/masks/Viral Pneumonia-70.png,True,G20612,test,65030adc92d0b634921782647adda586,d03d625f5e5a3928
Viral Pneumonia-700.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-700.png,Viral Pneumonia/masks/Viral Pneumonia-700.png,True,G20613,train,d0e7128f02ed8a2e99a6c65ee01b2bda,c06f5f5b49090d4f
Viral Pneumonia-701.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-701.png,Viral Pneumonia/masks/Viral Pneumonia-701.png,True,G20614,test,f37112f0bbc02a759c4d1e1157ddc4d8,953b7850510b3b7b
Viral Pneumonia-702.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-702.png,Viral Pneumonia/masks/Viral Pneumonia-702.png,True,G20615,train,0b50372f870c7fd3bea520e85186b015,d476c71bad4d20e4
Viral Pneumonia-703.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-703.png,Viral Pneumonia/masks/Viral Pneumonia-703.png,True,G20616,val,577ad585f91753bcdcc1a6f1116850d2,d16be71ed2423ac2
Viral Pneumonia-704.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-704.png,Viral Pneumonia/masks/Viral Pneumonia-704.png,True,G20617,train,57e49aeec1d571c524ccaa9631c4d74d,9029e29edac33fe0
Viral Pneumonia-705.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-705.png,Viral Pneumonia/masks/Viral Pneumonia-705.png,True,G20618,train,33c58fbcce257bcb9da2db9ffb527d59,d02b7f505a4a4b6b
Viral Pneumonia-706.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-706.png,Viral Pneumonia/masks/Viral Pneumonia-706.png,True,G20619,train,930a41cc3d8966106f0b9b92d6673ae0,9107c73ee469b06b
Viral Pneumonia-707.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-707.png,Viral Pneumonia/masks/Viral Pneumonia-707.png,True,G20620,train,ed9062d313cd0a812b1c963d250f7e3e,d0b7c63ac12db04f
Viral Pneumonia-708.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-708.png,Viral Pneumonia/masks/Viral Pneumonia-708.png,True,G20621,val,3cd91196e57375e96835373a5ce63b39,9428e31bc20ff46f
Viral Pneumonia-709.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-709.png,Viral Pneumonia/masks/Viral Pneumonia-709.png,True,G20622,train,d5b6c1799ede8fb4c382f970be24ae7e,812bc60e0ce87f7b
Viral Pneumonia-71.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-71.png,Viral Pneumonia/masks/Viral Pneumonia-71.png,True,G20623,train,2fc8823e8d6ecd73e714a50991243f89,957963262a0f3a3d
Viral Pneumonia-710.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-710.png,Viral Pneumonia/masks/Viral Pneumonia-710.png,True,G20624,train,fa0caa0d2fb208cf083875462ea27603,d035c33fe51f8a03
Viral Pneumonia-711.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-711.png,Viral Pneumonia/masks/Viral Pneumonia-711.png,True,G20625,train,87d290c073f433b67ee05e03422f382b,d465e73ba30ba30a
Viral Pneumonia-712.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-712.png,Viral Pneumonia/masks/Viral Pneumonia-712.png,True,G20626,train,7aae9491342337f24887b36819f1dd6e,812fc60e1c493dfb
Viral Pneumonia-713.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-713.png,Viral Pneumonia/masks/Viral Pneumonia-713.png,True,G20627,train,fedbf1e01ccfae1a9c3ce10f3d1eef75,d03fe629b88b3cb0
Viral Pneumonia-714.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-714.png,Viral Pneumonia/masks/Viral Pneumonia-714.png,True,G20628,val,5b47589ff8d9be59e0ededd525c821b5,d062c31bc33de06f
Viral Pneumonia-715.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-715.png,Viral Pneumonia/masks/Viral Pneumonia-715.png,True,G20629,train,da3116be3b14364563c04232fadcad36,812bc71e8eca7cf0
Viral Pneumonia-716.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-716.png,Viral Pneumonia/masks/Viral Pneumonia-716.png,True,G20630,train,dd7aa310e228b6abf86d8cd823094b8c,d43c924b9da8cb3c
Viral Pneumonia-717.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-717.png,Viral Pneumonia/masks/Viral Pneumonia-717.png,True,G20631,train,68d8c1e5d36fa2a231254b2cc4672a09,d035c21a80cbbfeb
Viral Pneumonia-718.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-718.png,Viral Pneumonia/masks/Viral Pneumonia-718.png,True,G20632,val,95a12a5b47b733ab5582e74b5ad83f7a,d02de23fe24bb843
Viral Pneumonia-719.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-719.png,Viral Pneumonia/masks/Viral Pneumonia-719.png,True,G20633,train,27a231b1b943bded37513c2fb3eac837,d4fc875ba51e210e
Viral Pneumonia-72.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-72.png,Viral Pneumonia/masks/Viral Pneumonia-72.png,True,G20634,test,1dd24be7f512fa767127a21904d2090b,9478e31f0707266f
Viral Pneumonia-720.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-720.png,Viral Pneumonia/masks/Viral Pneumonia-720.png,True,G20635,test,3cecc31afc812b949739f85a83f2cd3e,94eaa33f8f3e6802
Viral Pneumonia-721.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-721.png,Viral Pneumonia/masks/Viral Pneumonia-721.png,True,G20636,test,88af3c70649af84d7a32faa49fbe0698,917b5e5e5e19c109
Viral Pneumonia-722.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-722.png,Viral Pneumonia/masks/Viral Pneumonia-722.png,True,G20637,train,c8280784b61211e67a7f777059477355,914ac618817df9e7
Viral Pneumonia-723.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-723.png,Viral Pneumonia/masks/Viral Pneumonia-723.png,True,G20638,train,26efe14fd3c38bbb737fad1f9ee711f8,d035e33f864b38c3
Viral Pneumonia-724.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-724.png,Viral Pneumonia/masks/Viral Pneumonia-724.png,True,G20639,train,3db81dc16e4ceac7ae8a39eb3c7df06f,d064c65ac179a5cf
Viral Pneumonia-725.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-725.png,Viral Pneumonia/masks/Viral Pneumonia-725.png,True,G20640,train,c16ef59c2d9494f8a39078e23768eb19,c0eeb73db52d2803
Viral Pneumonia-726.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-726.png,Viral Pneumonia/masks/Viral Pneumonia-726.png,True,G20641,train,c589241cc9aa52ec1c9b4f0c82210549,9069c21e425e6f7d
Viral Pneumonia-727.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-727.png,Viral Pneumonia/masks/Viral Pneumonia-727.png,True,G20642,train,c67b12ded169868cf965ac9eb2e39e13,d036c30f9eed3cc0
Viral Pneumonia-728.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-728.png,Viral Pneumonia/masks/Viral Pneumonia-728.png,True,G20643,train,992f4338a6531639092644d96b2c5f66,d068c73b964e3d83
Viral Pneumonia-729.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-729.png,Viral Pneumonia/masks/Viral Pneumonia-729.png,True,G20644,train,b77c09f2ac141e35bb11698d1de3ab59,d0fae75fe9850884
Viral Pneumonia-73.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-73.png,Viral Pneumonia/masks/Viral Pneumonia-73.png,True,G20645,train,9e470ef79e7c99e822f31216ba1d7ea1,9429e322afe63d38
Viral Pneumonia-730.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-730.png,Viral Pneumonia/masks/Viral Pneumonia-730.png,True,G20646,train,b0f58b75b8729e60555828607a9a908e,d568822be96d6c4d
Viral Pneumonia-731.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-731.png,Viral Pneumonia/masks/Viral Pneumonia-731.png,True,G20647,train,5a46188071547a14fbdbbaa22e844aea,90648798e53ce73d
Viral Pneumonia-732.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-732.png,Viral Pneumonia/masks/Viral Pneumonia-732.png,True,G20648,train,af48c228101b2843ea2a1faa2ae9e7fb,d4f497398738433c
Viral Pneumonia-733.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-733.png,Viral Pneumonia/masks/Viral Pneumonia-733.png,True,G20649,val,5fb8b2cd28e23f6f5701efc448d9fbf9,91eaa73b851fa18a
Viral Pneumonia-734.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-734.png,Viral Pneumonia/masks/Viral Pneumonia-734.png,True,G20650,val,5bd5ac2f198c0688e6c9cea8c6952ffd,d066c61ac06db5cf
Viral Pneumonia-735.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-735.png,Viral Pneumonia/masks/Viral Pneumonia-735.png,True,G20651,val,645e50c845ffa8c2bcadcfdb5bc4b5e3,d434c21fe74b3cc1
Viral Pneumonia-736.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-736.png,Viral Pneumonia/masks/Viral Pneumonia-736.png,True,G20652,test,3010152069e13223377c025bda47230b,956aaabaea38c278
Viral Pneumonia-737.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-737.png,Viral Pneumonia/masks/Viral Pneumonia-737.png,True,G20653,test,afd343767e20181bdf27cd4aacf91b6b,913fd659f1313503
Viral Pneumonia-738.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-738.png,Viral Pneumonia/masks/Viral Pneumonia-738.png,True,G20654,train,9596b2d52ed6a108a2d3c0eae0adac2f,d076c31bc54d9947
Viral Pneumonia-739.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-739.png,Viral Pneumonia/masks/Viral Pneumonia-739.png,True,G20655,test,8093a2a71ddd77a1c4a1e9e4e62f2bfa,d1e1cf9e97189186
Viral Pneumonia-74.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-74.png,Viral Pneumonia/masks/Viral Pneumonia-74.png,True,G20656,train,1c28d2b6859a90eb21544f885109150d,d42f621f57573448
Viral Pneumonia-740.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-740.png,Viral Pneumonia/masks/Viral Pneumonia-740.png,True,G20657,train,c6b02f768d1d38fedb7091f0b1c6038e,d4f4d31be10f904b
Viral Pneumonia-741.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-741.png,Viral Pneumonia/masks/Viral Pneumonia-741.png,True,G20658,train,dd05421b151ae9b9103863e3f8393ff3,91a5c60f92cb3c7a
Viral Pneumonia-742.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-742.png,Viral Pneumonia/masks/Viral Pneumonia-742.png,True,G20659,train,fa498054e01f5fac9ff4183df70b45a3,982de51eb2c33bb0
Viral Pneumonia-743.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-743.png,Viral Pneumonia/masks/Viral Pneumonia-743.png,True,G20660,train,6ce6d21b347e7063ac620ee2089eb157,d007c31efc693cca
Viral Pneumonia-744.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-744.png,Viral Pneumonia/masks/Viral Pneumonia-744.png,True,G20661,train,cf20204c2ac6300f37d35d22c247c1d5,d465e33fe51f2804
Viral Pneumonia-745.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-745.png,Viral Pneumonia/masks/Viral Pneumonia-745.png,True,G20662,train,f64f3a847068ca5c19b427d6227e487a,8101aba0ee75f774
Viral Pneumonia-746.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-746.png,Viral Pneumonia/masks/Viral Pneumonia-746.png,True,G20663,train,359e658ecbcf6dcb2353996634d08b74,d06ac71e9dcb1d90
Viral Pneumonia-747.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-747.png,Viral Pneumonia/masks/Viral Pneumonia-747.png,True,G20664,test,da706fba03664c1b1086a522e6e305e5,d065c63ac33db04f
Viral Pneumonia-748.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-748.png,Viral Pneumonia/masks/Viral Pneumonia-748.png,True,G20665,train,3bcb21eb616cc84b23dfdae3d19d6cf1,9164863ac539957f
Viral Pneumonia-749.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-749.png,Viral Pneumonia/masks/Viral Pneumonia-749.png,True,G20666,test,2f791eb2232171b3567288f7c785063e,91a5c61a865ab6cf
Viral Pneumonia-75.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-75.png,Viral Pneumonia/masks/Viral Pneumonia-75.png,True,G20656,train,1c28d2b6859a90eb21544f885109150d,d42f621f57573448
Viral Pneumonia-750.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-750.png,Viral Pneumonia/masks/Viral Pneumonia-750.png,True,G20667,train,b52b5b160f0ddcf98b31941328f18dd5,9455e51e3ae22bac
Viral Pneumonia-751.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-751.png,Viral Pneumonia/masks/Viral Pneumonia-751.png,True,G20668,train,7d8f73366a83b7b543cc9957b50a2b20,d0e2975b9d788668
Viral Pneumonia-752.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-752.png,Viral Pneumonia/masks/Viral Pneumonia-752.png,True,G20669,train,18cd0f2d44e6875468c8c601d175ca8a,8133e336f4c03df4
Viral Pneumonia-753.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-753.png,Viral Pneumonia/masks/Viral Pneumonia-753.png,True,G20670,train,5c1260b6900a07d304e760dae2ade3b8,9105eb2fb152ad53
Viral Pneumonia-754.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-754.png,Viral Pneumonia/masks/Viral Pneumonia-754.png,True,G20671,val,91121656c3c383fd56ee7ede50b33cf8,d0f0d778e63da904
Viral Pneumonia-755.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-755.png,Viral Pneumonia/masks/Viral Pneumonia-755.png,True,G20672,val,7da778d6f8ed3e9b70437f1a2af28f5f,953cf47094333da5
Viral Pneumonia-756.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-756.png,Viral Pneumonia/masks/Viral Pneumonia-756.png,True,G20673,val,e302135f44a3bad2815616f7c60acd49,d521e21fee4bb640
Viral Pneumonia-757.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-757.png,Viral Pneumonia/masks/Viral Pneumonia-757.png,True,G20674,test,5215c74f09f5b8fb72c819331b42a2a6,91fde73eb8693800
Viral Pneumonia-758.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-758.png,Viral Pneumonia/masks/Viral Pneumonia-758.png,True,G20675,train,111f64fee076d023626d3042347e1c91,90efd67ac21ac51c
Viral Pneumonia-759.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-759.png,Viral Pneumonia/masks/Viral Pneumonia-759.png,True,G20676,train,49def0b3c6ae7fb9ae7263cef9b1035d,d42aeb0f1f4c1c5c
Viral Pneumonia-76.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-76.png,Viral Pneumonia/masks/Viral Pneumonia-76.png,True,G20677,train,44ee314f77b312f701fb3381bbbb17d4,947e430f2f3c2c6c
Viral Pneumonia-760.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-760.png,Viral Pneumonia/masks/Viral Pneumonia-760.png,True,G20678,train,c4c7f1fc7239b71337bed4af77774850,9c2fe71e4a693c44
Viral Pneumonia-761.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-761.png,Viral Pneumonia/masks/Viral Pneumonia-761.png,True,G20679,train,97c57bd56bebaec75066342793c88287,913bc65a5c427d4b
Viral Pneumonia-762.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-762.png,Viral Pneumonia/masks/Viral Pneumonia-762.png,True,G20680,val,9ab6673126fad6c86893ebcc7d64fb30,d47e871e0d0f6896
Viral Pneumonia-763.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-763.png,Viral Pneumonia/masks/Viral Pneumonia-763.png,True,G20681,train,67e5ac33be6b68eb44fd74397fa9d2d5,907e561e1a1b3da9
Viral Pneumonia-764.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-764.png,Viral Pneumonia/masks/Viral Pneumonia-764.png,True,G20682,train,ca81d5a40d52b29a417dff3bccbc4d67,907ca6e33ac74b38
Viral Pneumonia-765.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-765.png,Viral Pneumonia/masks/Viral Pneumonia-765.png,True,G20683,val,a865a4567c96fb7842529341daf4ad52,91ebeefac21ac448
Viral Pneumonia-766.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-766.png,Viral Pneumonia/masks/Viral Pneumonia-766.png,True,G20684,train,bc7f49adbaa03cba0f3645b88f047639,d025c71ee469bc6a
Viral Pneumonia-767.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-767.png,Viral Pneumonia/masks/Viral Pneumonia-767.png,True,G20685,train,600fe503e63275240c1fa95b0c0f6126,d03bf75b584a3a90
Viral Pneumonia-768.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-768.png,Viral Pneumonia/masks/Viral Pneumonia-768.png,True,G20686,train,17bde0799379ffbcede6a64b73abc990,9d64c429a57ec933
Viral Pneumonia-769.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-769.png,Viral Pneumonia/masks/Viral Pneumonia-769.png,True,G20687,train,ee8cd0c2aaffe49b6697de41ab8ce577,946fe73eaa38282c
Viral Pneumonia-77.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-77.png,Viral Pneumonia/masks/Viral Pneumonia-77.png,True,G20688,train,362fd131c97d46a00a059668bd34bdc8,953ae33f810f007f
Viral Pneumonia-770.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-770.png,Viral Pneumonia/masks/Viral Pneumonia-770.png,True,G20689,train,3478df5d05ba9dd2ecb722fea5526fb8,9079e965adc4c6b2
Viral Pneumonia-771.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-771.png,Viral Pneumonia/masks/Viral Pneumonia-771.png,True,G20690,train,22311dca9f9458b325158530548dcfdc,947ea706031f3d5c
Viral Pneumonia-772.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-772.png,Viral Pneumonia/masks/Viral Pneumonia-772.png,True,G20691,train,8aea6c9c25cef9d93d1440acb077bef9,d52ae7049ec23b1d
Viral Pneumonia-773.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-773.png,Viral Pneumonia/masks/Viral Pneumonia-773.png,True,G20692,val,84ae02d329bbd11711f1af4cedaa9045,913be72d9cc6a50c
Viral Pneumonia-774.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-774.png,Viral Pneumonia/masks/Viral Pneumonia-774.png,True,G20693,test,3252495edf5b4587e2b46969c995531e,91e0873b871a5bce
Viral Pneumonia-775.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-775.png,Viral Pneumonia/masks/Viral Pneumonia-775.png,True,G20694,test,aca9cdceaa10ee28584b0b5b7711b2b9,d4f4b31be33fa004
Viral Pneumonia-776.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-776.png,Viral Pneumonia/masks/Viral Pneumonia-776.png,True,G20695,test,a4a247bc1a1ad8dec437f5552e9b0f95,d07ec73eb10f1981
Viral Pneumonia-777.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-777.png,Viral Pneumonia/masks/Viral Pneumonia-777.png,True,G20696,train,37c6d17d839dddf4ae81fde782098ebc,9029c71eea4fb8c3
Viral Pneumonia-778.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-778.png,Viral Pneumonia/masks/Viral Pneumonia-778.png,True,G20697,test,7faf299d7694275f19a7bb09e13e91c6,912bc61ed64b3c4b
Viral Pneumonia-779.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-779.png,Viral Pneumonia/masks/Viral Pneumonia-779.png,True,G20698,train,eeb0e231e65ed2f2a0ec1d40cbc3468d,9161a73af04e7996
Viral Pneumonia-78.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-78.png,Viral Pneumonia/masks/Viral Pneumonia-78.png,True,G20699,train,694cb99a9c6cfc4630e0677a1ee4540a,95ffe22ce61c300d
Viral Pneumonia-780.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-780.png,Viral Pneumonia/masks/Viral Pneumonia-780.png,True,G20700,train,fb5d18077a6b4df875fa682f6e3f2ba1,9139e61a829e3ec7
Viral Pneumonia-781.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-781.png,Viral Pneumonia/masks/Viral Pneumonia-781.png,True,G20701,train,83170fdaa7d58f598e50e8b9c71fe7c1,d464c71ac39e4997
Viral Pneumonia-782.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-782.png,Viral Pneumonia/masks/Viral Pneumonia-782.png,True,G20702,val,87146cba4f707d35fc82d6de3db9537d,dc6ce0126b966d9a
Viral Pneumonia-783.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-783.png,Viral Pneumonia/masks/Viral Pneumonia-783.png,True,G20703,train,e75dcd17c2c610a50076ae105468ce9a,d07ae71f49972883
Viral Pneumonia-784.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-784.png,Viral Pneumonia/masks/Viral Pneumonia-784.png,True,G20704,train,9b21f06e4c50efd411d090d37f257512,c036964b976b9dc2
Viral Pneumonia-785.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-785.png,Viral Pneumonia/masks/Viral Pneumonia-785.png,True,G20705,train,19c7250a815a2b89e6f0e8edea807ec0,946fa61a3d3ca43c
Viral Pneumonia-786.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-786.png,Viral Pneumonia/masks/Viral Pneumonia-786.png,True,G20706,test,4e46ccfa2950c9cae48a8b9e37d8954c,806fee7e4a591c49
Viral Pneumonia-787.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-787.png,Viral Pneumonia/masks/Viral Pneumonia-787.png,True,G20707,test,6b7281b722c1849fb29a45fd4af95396,d06b471e59493d2d
Viral Pneumonia-788.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-788.png,Viral Pneumonia/masks/Viral Pneumonia-788.png,True,G20708,train,b7c791a0c39f2bbadcc489ac04e8ad7b,c0ecef1913393c4e
Viral Pneumonia-789.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-789.png,Viral Pneumonia/masks/Viral Pneumonia-789.png,True,G20709,train,382907e3cd3f825979ec608858967bb5,946ac31a81347def
Viral Pneumonia-79.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-79.png,Viral Pneumonia/masks/Viral Pneumonia-79.png,True,G20710,train,2a3a1837e4feaf8a25eedc33e0826e6a,9125c61aee4bbc63
Viral Pneumonia-790.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-790.png,Viral Pneumonia/masks/Viral Pneumonia-790.png,True,G20711,train,638f40d0cb7de7f96e2e71e69fea755b,9538f0413de5c1b7
Viral Pneumonia-791.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-791.png,Viral Pneumonia/masks/Viral Pneumonia-791.png,True,G20712,train,05c9eae62c2a5751186426fa50a4140b,d07ac779819a6996
Viral Pneumonia-792.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-792.png,Viral Pneumonia/masks/Viral Pneumonia-792.png,True,G20713,val,5c073c67398c99dfb7fe371a14549916,9829e55fb5c21cf0
Viral Pneumonia-793.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-793.png,Viral Pneumonia/masks/Viral Pneumonia-793.png,True,G20714,val,97b6c3d64e5030467753d9bf26050d3e,d0ffef1a820f18c1
Viral Pneumonia-794.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-794.png,Viral Pneumonia/masks/Viral Pneumonia-794.png,True,G20715,train,bc7a330788c8270781eb033f7ce7777e,947ea71f8739091c
Viral Pneumonia-795.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-795.png,Viral Pneumonia/masks/Viral Pneumonia-795.png,True,G20716,train,142ad54b8eedb2a44599bd5424f79b39,90bf47120e5f1f1c
Viral Pneumonia-796.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-796.png,Viral Pneumonia/masks/Viral Pneumonia-796.png,True,G20717,train,0dcff59f11606a24ee1d7264640bfd1a,d4bf5542aa2f28aa
Viral Pneumonia-797.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-797.png,Viral Pneumonia/masks/Viral Pneumonia-797.png,True,G20718,train,8d26815be4432e59f7024fab62b2cb21,c074e61b93cf9b81
Viral Pneumonia-798.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-798.png,Viral Pneumonia/masks/Viral Pneumonia-798.png,True,G20719,train,a52b9998f342226b5e723b7fb0a58682,9169e73e8e0bb2c2
Viral Pneumonia-799.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-799.png,Viral Pneumonia/masks/Viral Pneumonia-799.png,True,G20720,train,00effedcf1b4a01b5228660a6dac0e3c,d525e21eea4bb443
Viral Pneumonia-8.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-8.png,Viral Pneumonia/masks/Viral Pneumonia-8.png,True,G20721,train,31e8a3a80b980ce7c34980975683c9b3,d025c21bc76f386b
Viral Pneumonia-80.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-80.png,Viral Pneumonia/masks/Viral Pneumonia-80.png,True,G20722,train,d32d83df89145089af6e17c2aad135c3,c9bf565a1c491d49
Viral Pneumonia-800.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-800.png,Viral Pneumonia/masks/Viral Pneumonia-800.png,True,G20723,train,6470f5ab595898d9a69615fd26c4a86c,d424e21bc34b9ecb
Viral Pneumonia-801.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-801.png,Viral Pneumonia/masks/Viral Pneumonia-801.png,True,G20724,train,e2fa34e38a4e7813f637482520175e85,c06f971b4b87c343
Viral Pneumonia-802.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-802.png,Viral Pneumonia/masks/Viral Pneumonia-802.png,True,G20725,train,5206028da8034b44eb84d34a195c98f8,807de65abccb3914
Viral Pneumonia-803.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-803.png,Viral Pneumonia/masks/Viral Pneumonia-803.png,True,G20726,train,21627ba7f467c9157b2f56a89b5885be,c07d2d2c6eaee44a
Viral Pneumonia-804.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-804.png,Viral Pneumonia/masks/Viral Pneumonia-804.png,True,G20727,val,808f41777aa1e4e8a7c0ae7b45728b41,d2bae7bee61b8004
Viral Pneumonia-805.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-805.png,Viral Pneumonia/masks/Viral Pneumonia-805.png,True,G20728,val,6110e4c1204ed22f73dafa8c7f139794,c16f365a9b5a5a05
Viral Pneumonia-806.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-806.png,Viral Pneumonia/masks/Viral Pneumonia-806.png,True,G20729,train,93fbf617ccfdfdf4034e535e03df1c13,c06b2d4bad5eb514
Viral Pneumonia-807.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-807.png,Viral Pneumonia/masks/Viral Pneumonia-807.png,True,G20730,test,d49150421b6e4e990f02e05c4945c27b,d464c31e93cb39c5
Viral Pneumonia-808.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-808.png,Viral Pneumonia/masks/Viral Pneumonia-808.png,True,G20731,train,1b2bbdbbe7514b430436632076315f2e,807ab41eb23f0d3d
Viral Pneumonia-809.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-809.png,Viral Pneumonia/masks/Viral Pneumonia-809.png,True,G20732,val,53c823620a1b2ec48427bbfe3d797e67,9176865a8f39873c
Viral Pneumonia-81.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-81.png,Viral Pneumonia/masks/Viral Pneumonia-81.png,True,G20733,test,72d4a0d60fd3d3ce1fd4b5fdef2da4c2,d074c21beb2bb983
Viral Pneumonia-810.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-810.png,Viral Pneumonia/masks/Viral Pneumonia-810.png,True,G20734,test,2dbc9c73e2e3723a398bc5b98275cb4d,d039e20bbcca1fb4
Viral Pneumonia-811.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-811.png,Viral Pneumonia/masks/Viral Pneumonia-811.png,True,G20735,train,b1a5e8da6906646a3fc2e73a9cdacb33,916be73a804a39ed
Viral Pneumonia-812.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-812.png,Viral Pneumonia/masks/Viral Pneumonia-812.png,True,G20736,train,33d0df08a605b11774d1c4c3105307e6,9175c61eb94bbc42
Viral Pneumonia-813.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-813.png,Viral Pneumonia/masks/Viral Pneumonia-813.png,True,G20737,train,587e9d1fbbd735db8fb34ccc3cc04574,d36ac739c4393196
Viral Pneumonia-814.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-814.png,Viral Pneumonia/masks/Viral Pneumonia-814.png,True,G20738,val,cc7de6b7658c4b6dc3627776ce748cad,d0ffe71a81833992
Viral Pneumonia-815.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-815.png,Viral Pneumonia/masks/Viral Pneumonia-815.png,True,G20739,val,1f00e3f8fae05f9f3370f050f7772e01,98e1c578c6798769
Viral Pneumonia-816.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-816.png,Viral Pneumonia/masks/Viral Pneumonia-816.png,True,G20740,train,65d1caac511e5f7a439961b198fe8b05,9470e75aa95f6825
Viral Pneumonia-817.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-817.png,Viral Pneumonia/masks/Viral Pneumonia-817.png,True,G20741,train,fab411a0bdc795904799b5239d31bb22,d46ca43eb41af187
Viral Pneumonia-818.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-818.png,Viral Pneumonia/masks/Viral Pneumonia-818.png,True,G20742,test,f6857d244f1624077f851a145fbce698,81d4e69eda4f9825
Viral Pneumonia-819.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-819.png,Viral Pneumonia/masks/Viral Pneumonia-819.png,True,G20743,train,0f3ca25b55074d9986943d2fa10d8a7e,d47ce31bc3874985
Viral Pneumonia-82.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-82.png,Viral Pneumonia/masks/Viral Pneumonia-82.png,True,G20744,val,195a24eb6985a65c990c017f90958eda,91fdc73a865a304d
Viral Pneumonia-820.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-820.png,Viral Pneumonia/masks/Viral Pneumonia-820.png,True,G20745,train,6f08c1d81ce961cc796d146b1978ce97,d46ce71afa832a83
Viral Pneumonia-821.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-821.png,Viral Pneumonia/masks/Viral Pneumonia-821.png,True,G20746,test,cad143d1d209e9f7cb8dc6ca5c1f1172,d02de54abed24b98
Viral Pneumonia-822.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-822.png,Viral Pneumonia/masks/Viral Pneumonia-822.png,True,G20747,test,f9420734d06d6703855eae138bb84256,d1b5c21be96c3cc4
Viral Pneumonia-823.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-823.png,Viral Pneumonia/masks/Viral Pneumonia-823.png,True,G20748,train,8a305e9f4f86228a380940149557f335,91b1e70fbece8c10
Viral Pneumonia-824.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-824.png,Viral Pneumonia/masks/Viral Pneumonia-824.png,True,G20749,train,34d83a29c68468421e6e63579cc9fc96,d42f6c1346525b5b
Viral Pneumonia-825.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-825.png,Viral Pneumonia/masks/Viral Pneumonia-825.png,True,G20750,train,51933f6331ba4c7bef57438439166cd9,912dc61aa8ca3eeb
Viral Pneumonia-826.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-826.png,Viral Pneumonia/masks/Viral Pneumonia-826.png,True,G20751,train,e5dd722a93fd06eb7b8cbd390984d710,db69a9e1a83ce0b4
Viral Pneumonia-827.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-827.png,Viral Pneumonia/masks/Viral Pneumonia-827.png,True,G20752,train,d529e626da16e0e8cfc120e7430ee03d,c06ab7b7ab2d2505
Viral Pneumonia-828.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-828.png,Viral Pneumonia/masks/Viral Pneumonia-828.png,True,G20753,train,0fd6d583d3ff16b0374339658dd42b6f,c06f3191117d7576
Viral Pneumonia-829.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-829.png,Viral Pneumonia/masks/Viral Pneumonia-829.png,True,G20754,train,998522c3cd75a080d58c4effdec99568,d474c21bc729a34f
Viral Pneumonia-83.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-83.png,Viral Pneumonia/masks/Viral Pneumonia-83.png,True,G20755,val,6ca4ce32688f283081064f6540d692fa,d0e4e73eed3c2442
Viral Pneumonia-830.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-830.png,Viral Pneumonia/masks/Viral Pneumonia-830.png,True,G20756,train,0a34e959933755100f282a4c2748ded2,c02fa32db8e93cf0
Viral Pneumonia-831.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-831.png,Viral Pneumonia/masks/Viral Pneumonia-831.png,True,G20757,train,dd03ea29b05ad27893e8452dd28094d6,c0adb61f530f4b0e
Viral Pneumonia-832.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-832.png,Viral Pneumonia/masks/Viral Pneumonia-832.png,True,G20758,train,ee34b32e9c957c70b147fe74a4997cd6,8568e7b65a181d3d
Viral Pneumonia-833.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-833.png,Viral Pneumonia/masks/Viral Pneumonia-833.png,True,G20759,test,1de9dec58b438d361fbfb8d77c90f7e0,d0a5c33bdc693cc2
Viral Pneumonia-834.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-834.png,Viral Pneumonia/masks/Viral Pneumonia-834.png,True,G20760,val,b939d87b572f15ea6dff2a8f9db83114,902ee33be66a3ce0
Viral Pneumonia-835.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-835.png,Viral Pneumonia/masks/Viral Pneumonia-835.png,True,G20761,train,dae715f9fdd3fe5e55425b37a7453d02,d02fe70a29493d7c
Viral Pneumonia-836.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-836.png,Viral Pneumonia/masks/Viral Pneumonia-836.png,True,G20762,train,3360e68ae245f87a1c1ed3a1f1211540,d042c31fed4dbcc4
Viral Pneumonia-837.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-837.png,Viral Pneumonia/masks/Viral Pneumonia-837.png,True,G20763,train,7f0ff4a1030c1950c70c4363e6cf5c75,d072c77ac73d8186
Viral Pneumonia-838.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-838.png,Viral Pneumonia/masks/Viral Pneumonia-838.png,True,G20764,train,b9b07a075b07c8671065cbfc9bd6cc42,902be35eeb4b2c61
Viral Pneumonia-839.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-839.png,Viral Pneumonia/masks/Viral Pneumonia-839.png,True,G20765,train,afe89e0c6d349d751ec2ea4fd3c94601,91e5c67a864e39c6
Viral Pneumonia-84.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-84.png,Viral Pneumonia/masks/Viral Pneumonia-84.png,True,G20766,test,9e1cc2f1b637296174348ce6106959b2,d0ffa70d01115d57
Viral Pneumonia-840.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-840.png,Viral Pneumonia/masks/Viral Pneumonia-840.png,True,G20767,train,6671ec5ed3cf6355fb9eb632f8602c5c,d02de23f68693d49
Viral Pneumonia-841.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-841.png,Viral Pneumonia/masks/Viral Pneumonia-841.png,True,G20768,train,09063ec35f736da587ec3fc243baf385,d5a9c20be9693ce1
Viral Pneumonia-842.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-842.png,Viral Pneumonia/masks/Viral Pneumonia-842.png,True,G20769,train,3c1b08806f4ff61db7467e7575ab9591,d13bf7923838213b
Viral Pneumonia-843.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-843.png,Viral Pneumonia/masks/Viral Pneumonia-843.png,True,G20770,train,72ab5bf44c68b6f57aa2ab04eaa34a97,d101c71efe583dc8
Viral Pneumonia-844.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-844.png,Viral Pneumonia/masks/Viral Pneumonia-844.png,True,G20771,train,242b84fd6f3a739ec7d7a9c5fcf3f616,d42bcb1ffc4d3840
Viral Pneumonia-845.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-845.png,Viral Pneumonia/masks/Viral Pneumonia-845.png,True,G20772,train,18a29656767e12735f3633f5ec9f719e,d0b3ea3ed25cd045
Viral Pneumonia-846.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-846.png,Viral Pneumonia/masks/Viral Pneumonia-846.png,True,G20773,val,8fae6c8d3cc002dd3131638c3c641f77,917e871a8f0f198e
Viral Pneumonia-847.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-847.png,Viral Pneumonia/masks/Viral Pneumonia-847.png,True,G20774,val,bffe0486efc5d08761ca66cbef27afed,c0f4c31bd14f3cd2
Viral Pneumonia-848.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-848.png,Viral Pneumonia/masks/Viral Pneumonia-848.png,True,G20775,test,a63dd9488c1a4c7cc06f7a86950cc64d,9025e71ee35bbc42
Viral Pneumonia-849.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-849.png,Viral Pneumonia/masks/Viral Pneumonia-849.png,True,G20776,test,cd474afd231ba9e135576d51df2cb7c2,d07ea73f8d072924
Viral Pneumonia-85.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-85.png,Viral Pneumonia/masks/Viral Pneumonia-85.png,True,G20777,train,cd6ec754948ed0646a090b51b7ced329,d07cfe3ead3c0038
Viral Pneumonia-850.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-850.png,Viral Pneumonia/masks/Viral Pneumonia-850.png,True,G20778,train,0e871c30a3e33b89dc0834d7d2887869,903de71ac22b78c7
Viral Pneumonia-851.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-851.png,Viral Pneumonia/masks/Viral Pneumonia-851.png,True,G20779,train,c00aee71d053c640e5b920ccee8beaab,9127e63ae86b3463
Viral Pneumonia-852.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-852.png,Viral Pneumonia/masks/Viral Pneumonia-852.png,True,G20780,train,33b24d8700096f9cd90b8f0506b99a99,91e4c61ac75ab1ce
Viral Pneumonia-853.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-853.png,Viral Pneumonia/masks/Viral Pneumonia-853.png,True,G20781,train,7d45ad6fca06c7829316021e5ac6f97e,d474c63ded1c480f
Viral Pneumonia-854.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-854.png,Viral Pneumonia/masks/Viral Pneumonia-854.png,True,G20782,train,820a4ed489f2de6eaa56ee086a7597b0,d078e74b9f810b39
Viral Pneumonia-855.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-855.png,Viral Pneumonia/masks/Viral Pneumonia-855.png,True,G20783,train,55f8b5b4005413f010cd779e7bf1d27a,d02ff34b4f413c2c
Viral Pneumonia-856.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-856.png,Viral Pneumonia/masks/Viral Pneumonia-856.png,True,G20784,train,b5e61ae943e56bc7a3abc9d80cb7b9e9,dd6da41efa864aa0
Viral Pneumonia-857.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-857.png,Viral Pneumonia/masks/Viral Pneumonia-857.png,True,G20785,test,8b6e11038ae3455d11aca5ac446cc384,d0b5e33deb3c202c
Viral Pneumonia-858.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-858.png,Viral Pneumonia/masks/Viral Pneumonia-858.png,True,G20786,val,a549834d0ebf6e050e4b74442dd6029d,81a3e63afc6b3c48
Viral Pneumonia-859.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-859.png,Viral Pneumonia/masks/Viral Pneumonia-859.png,True,G20787,train,a4144aac30f6be36dda128efb5dcf71c,d424c20bb5e93cf6
Viral Pneumonia-86.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-86.png,Viral Pneumonia/masks/Viral Pneumonia-86.png,True,G20788,train,b10a2560e8b0d5d87e45f9b6f3e3c979,9121e71eea6a3d49
Viral Pneumonia-860.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-860.png,Viral Pneumonia/masks/Viral Pneumonia-860.png,True,G20789,train,6fcceffd2a3e312d86bc45ef75683627,9125c73ec23ed44e
Viral Pneumonia-861.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-861.png,Viral Pneumonia/masks/Viral Pneumonia-861.png,True,G20790,train,1aa991aaa21b85b6efab9fb3dd65ea75,c1a5e20be8683eeb
Viral Pneumonia-862.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-862.png,Viral Pneumonia/masks/Viral Pneumonia-862.png,True,G20791,val,51350562d911a69dba039fe92891d7ec,d4ffa30b830f6883
Viral Pneumonia-863.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-863.png,Viral Pneumonia/masks/Viral Pneumonia-863.png,True,G20792,train,3c3d7bf2138b06c55940ea1abe82fb34,d4e5c31ac31f88c7
Viral Pneumonia-864.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-864.png,Viral Pneumonia/masks/Viral Pneumonia-864.png,True,G20793,train,0fa748411cf3b3d7eb5edd7563cf08d9,d13bd65a5c541c95
Viral Pneumonia-865.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-865.png,Viral Pneumonia/masks/Viral Pneumonia-865.png,True,G20794,val,9ef576384cc606be926bba198202ed5c,9065c71aa139fc6d
Viral Pneumonia-866.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-866.png,Viral Pneumonia/masks/Viral Pneumonia-866.png,True,G20795,test,335fa070a6bd3c118f3747bd06d8316b,9125c71ec65ab64b
Viral Pneumonia-867.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-867.png,Viral Pneumonia/masks/Viral Pneumonia-867.png,True,G20796,train,9d39e8a01e8c58b8b014d88f4fade86b,d4f4a73aa70b2986
Viral Pneumonia-868.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-868.png,Viral Pneumonia/masks/Viral Pneumonia-868.png,True,G20797,train,04caa4269fef0295ef7e89fcea654119,d460e31be51b838f
Viral Pneumonia-869.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-869.png,Viral Pneumonia/masks/Viral Pneumonia-869.png,True,G20798,test,f1d1c1f15ae99463f4905e37847e3069,d074c71f95c33d84
Viral Pneumonia-87.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-87.png,Viral Pneumonia/masks/Viral Pneumonia-87.png,True,G20799,train,74a6b0ad88e7f407f934578cb23fd704,d035c70fbc4b3d42
Viral Pneumonia-870.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-870.png,Viral Pneumonia/masks/Viral Pneumonia-870.png,True,G20800,train,f81f3ff2bd327cc077c4c2d82e90b994,8125c61af44fbce6
Viral Pneumonia-871.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-871.png,Viral Pneumonia/masks/Viral Pneumonia-871.png,True,G20801,train,58b8c7b94eed0b912098f12922bf5dd7,9729c67a78ca7852
Viral Pneumonia-872.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-872.png,Viral Pneumonia/masks/Viral Pneumonia-872.png,True,G20802,train,ce55dde65f69230e437960569e78de53,953de67bb4ca2890
Viral Pneumonia-873.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-873.png,Viral Pneumonia/masks/Viral Pneumonia-873.png,True,G20803,train,3e8b30aa247384af6e66d6000ad800e9,d0f4873ba33ba146
Viral Pneumonia-874.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-874.png,Viral Pneumonia/masks/Viral Pneumonia-874.png,True,G20804,train,4ff9312a3056f174932aac5747c738bb,d070e71ac74fb00f
Viral Pneumonia-875.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-875.png,Viral Pneumonia/masks/Viral Pneumonia-875.png,True,G20805,train,13bee815493f7b7354ce1b466ee23a56,c06fbe3a0b055d35
Viral Pneumonia-876.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-876.png,Viral Pneumonia/masks/Viral Pneumonia-876.png,True,G20806,train,9cde2a80fa21226a3706b903fe35ad90,d07ee73fe13d2800
Viral Pneumonia-877.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-877.png,Viral Pneumonia/masks/Viral Pneumonia-877.png,True,G20807,train,9e62471acbc66262ef86a2138904ac95,d0e4c35ac739871b
Viral Pneumonia-878.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-878.png,Viral Pneumonia/masks/Viral Pneumonia-878.png,True,G20808,train,273a262d038fa37e5e8ce1bea27ce3b1,d0b4c30f9ecb3c98
Viral Pneumonia-879.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-879.png,Viral Pneumonia/masks/Viral Pneumonia-879.png,True,G20809,train,092991ddd8610e800b16dd4d20951504,9475e13ee64b98c2
Viral Pneumonia-88.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-88.png,Viral Pneumonia/masks/Viral Pneumonia-88.png,True,G20810,train,a09528a4205d2e882ba1ed3d96f95348,9125c61ef84bbcca
Viral Pneumonia-880.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-880.png,Viral Pneumonia/masks/Viral Pneumonia-880.png,True,G20811,val,884e2c95ef26426b1f6876c212cea308,d025c21be14b9cef
Viral Pneumonia-881.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-881.png,Viral Pneumonia/masks/Viral Pneumonia-881.png,True,G20812,val,1ec21082ef20b6213b051e726d6a3f3a,d4afc74d8629870e
Viral Pneumonia-882.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-882.png,Viral Pneumonia/masks/Viral Pneumonia-882.png,True,G20813,val,95b053a6816b352333b4c5cdc6f3a71c,913fc67ad8597051
Viral Pneumonia-883.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-883.png,Viral Pneumonia/masks/Viral Pneumonia-883.png,True,G20814,test,6807ff4c944a92ad70c8e8693af2ed3e,90edeeba8519954a
Viral Pneumonia-884.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-884.png,Viral Pneumonia/masks/Viral Pneumonia-884.png,True,G20815,test,b2b4e44e4fe616ce6a25916985c78613,d02ac74a7d697958
Viral Pneumonia-885.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-885.png,Viral Pneumonia/masks/Viral Pneumonia-885.png,True,G20816,train,211a63d2e2b5013e12b2d24ed4783610,d0f4e71f930b3883
Viral Pneumonia-886.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-886.png,Viral Pneumonia/masks/Viral Pneumonia-886.png,True,G20817,train,93eba687b2758b2f8cb91427b624296f,91e59e9a1b392c39
Viral Pneumonia-887.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-887.png,Viral Pneumonia/masks/Viral Pneumonia-887.png,True,G20818,train,3c83c3a6d9cdb34e46a335436f093be6,94e4c79fc718824f
Viral Pneumonia-888.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-888.png,Viral Pneumonia/masks/Viral Pneumonia-888.png,True,G20819,train,24f5c5357776a8327421e46eb989b47f,812bc21adcc83dfb
Viral Pneumonia-889.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-889.png,Viral Pneumonia/masks/Viral Pneumonia-889.png,True,G20820,train,5ed14a86f36c862b76d35f96bc4a7412,d0eecb1bc54d104f
Viral Pneumonia-89.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-89.png,Viral Pneumonia/masks/Viral Pneumonia-89.png,True,G20821,train,740e43638eef2c71f791884db478ef71,c42ec71bd9466cf0
Viral Pneumonia-890.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-890.png,Viral Pneumonia/masks/Viral Pneumonia-890.png,True,G20822,train,b160ceb720fd79f835b2ca7a0de119f0,d06ae75be9856861
Viral Pneumonia-891.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-891.png,Viral Pneumonia/masks/Viral Pneumonia-891.png,True,G20823,train,4099a78177a2c7f14db454d7a8759389,d027873ba56f304b
Viral Pneumonia-892.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-892.png,Viral Pneumonia/masks/Viral Pneumonia-892.png,True,G20824,test,120796c680555d7ba867ccb49bbe51b6,9070c71a851ff1de
Viral Pneumonia-893.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-893.png,Viral Pneumonia/masks/Viral Pneumonia-893.png,True,G20825,test,3ad737c67aa31846e561a0afad2e5bcc,9565e77a924a30c7
Viral Pneumonia-894.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-894.png,Viral Pneumonia/masks/Viral Pneumonia-894.png,True,G20826,train,a15b40e79e5fbb81bb74e1b4b73bb186,8071e36f3dc54e38
Viral Pneumonia-895.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-895.png,Viral Pneumonia/masks/Viral Pneumonia-895.png,True,G20827,train,8e479638f7257fb6becbfc3341059c91,912e061e1f7bd552
Viral Pneumonia-896.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-896.png,Viral Pneumonia/masks/Viral Pneumonia-896.png,True,G20828,train,baf322ffa57b91f776e4ad44f9a8bf30,c034e33d942d3ccf
Viral Pneumonia-897.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-897.png,Viral Pneumonia/masks/Viral Pneumonia-897.png,True,G20829,train,0ca23db698c8c9cdae03527fff793cb7,c0629718853de77e
Viral Pneumonia-898.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-898.png,Viral Pneumonia/masks/Viral Pneumonia-898.png,True,G20830,val,43bf4ad017586506827cbca0579ece64,803561c7377c36cb
Viral Pneumonia-899.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-899.png,Viral Pneumonia/masks/Viral Pneumonia-899.png,True,G20831,train,86cf0d4463bd2da5cb5c852252a41d35,d06de75ae96b3840
Viral Pneumonia-9.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-9.png,Viral Pneumonia/masks/Viral Pneumonia-9.png,True,G20832,train,0c157917f6dfa5060ff82684e3621ecf,9969871ac21ec2df
Viral Pneumonia-90.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-90.png,Viral Pneumonia/masks/Viral Pneumonia-90.png,True,G20833,train,d9c88381c2ecb86a5e0e9f655528a989,91a5c71ee01ff44a
Viral Pneumonia-900.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-900.png,Viral Pneumonia/masks/Viral Pneumonia-900.png,True,G20834,val,c95088cb5dbbac96ae32500863aaa4b0,81a9e20ebeca1ef8
Viral Pneumonia-901.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-901.png,Viral Pneumonia/masks/Viral Pneumonia-901.png,True,G20835,val,d3a5b18a6e62eeac34ed87447c6effa4,93bf8689b0e5b8b0
Viral Pneumonia-902.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-902.png,Viral Pneumonia/masks/Viral Pneumonia-902.png,True,G20836,test,3f9153c234a972fd70cac07cc9c1ba5b,9408e32d3e97171f
Viral Pneumonia-903.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-903.png,Viral Pneumonia/masks/Viral Pneumonia-903.png,True,G20837,train,9dd24803898f15cd65580222162cd680,d066c31fe54b944b
Viral Pneumonia-904.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-904.png,Viral Pneumonia/masks/Viral Pneumonia-904.png,True,G20838,train,95c5704c997b29b3e9f1902922aaaa1c,d1b9e63bf64b3002
Viral Pneumonia-905.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-905.png,Viral Pneumonia/masks/Viral Pneumonia-905.png,True,G20839,train,1644bb360a97248937da5d1b642309b0,d0fece38296c38c6
Viral Pneumonia-906.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-906.png,Viral Pneumonia/masks/Viral Pneumonia-906.png,True,G20840,train,aad71cf0004221f2847b7383897fe732,d02fe30bad693c64
Viral Pneumonia-907.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-907.png,Viral Pneumonia/masks/Viral Pneumonia-907.png,True,G20841,train,259fbcad54cd92d17c8f344ca438ada4,90f5c71aa54eb147
Viral Pneumonia-908.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-908.png,Viral Pneumonia/masks/Viral Pneumonia-908.png,True,G20842,train,c9921ddc11276a199af740ec0d6d79a1,d02ca32bb9cd3dd0
Viral Pneumonia-909.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-909.png,Viral Pneumonia/masks/Viral Pneumonia-909.png,True,G20843,train,8ce59c0309220b3d9c8a008d8a9593d0,90aecf5f540a3ac3
Viral Pneumonia-91.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-91.png,Viral Pneumonia/masks/Viral Pneumonia-91.png,True,G20844,train,a45ff7db1d593d2e33924e178807e8a5,d478a31be33da107
Viral Pneumonia-910.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-910.png,Viral Pneumonia/masks/Viral Pneumonia-910.png,True,G20845,train,b3ac9401a79f6d617e0706f0da400f6d,d01ac31f966bb4c3
Viral Pneumonia-911.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-911.png,Viral Pneumonia/masks/Viral Pneumonia-911.png,True,G20846,train,52076acab05d24a7568991a96dc26f61,81fcbe1a8f198e19
Viral Pneumonia-912.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-912.png,Viral Pneumonia/masks/Viral Pneumonia-912.png,True,G20847,train,dc187b3d4163b7f48f144bc13b14ee19,c07ffa5a5b580d48
Viral Pneumonia-913.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-913.png,Viral Pneumonia/masks/Viral Pneumonia-913.png,True,G20848,train,2c03c212c71d0e93275f1e5876d75c9e,d03fef3aba493810
Viral Pneumonia-914.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-914.png,Viral Pneumonia/masks/Viral Pneumonia-914.png,True,G20849,test,e434d830fa95e3469def6ffbe55f1205,d0bee33fb8693840
Viral Pneumonia-915.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-915.png,Viral Pneumonia/masks/Viral Pneumonia-915.png,True,G20850,train,6ff16370b9a72cdbdf0722ce56213e15,d025e73fbe691c40
Viral Pneumonia-916.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-916.png,Viral Pneumonia/masks/Viral Pneumonia-916.png,True,G20851,val,ff8f67cccb588009d5cd591220b60ddb,d0e5c31ec71bc44b
Viral Pneumonia-917.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-917.png,Viral Pneumonia/masks/Viral Pneumonia-917.png,True,G20852,val,1f189a5244b66882c812895ef4c6cd42,9565c71ac98f7885
Viral Pneumonia-918.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-918.png,Viral Pneumonia/masks/Viral Pneumonia-918.png,True,G20853,test,d6fb65b2f4aa08f9e144f7def25408e9,910de5433ff6c20b
Viral Pneumonia-919.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-919.png,Viral Pneumonia/masks/Viral Pneumonia-919.png,True,G20854,test,cd2d437b514aa131ed66d0b6c5e9a049,d0f7e71a860b3187
Viral Pneumonia-92.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-92.png,Viral Pneumonia/masks/Viral Pneumonia-92.png,True,G20855,val,a3f53f0f9077a3e36b4fa3267c833168,ddef821acf1c0d0c
Viral Pneumonia-920.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-920.png,Viral Pneumonia/masks/Viral Pneumonia-920.png,True,G20856,test,867a9447896772b9ba15828b356b69ff,913be76e3cc64614
Viral Pneumonia-921.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-921.png,Viral Pneumonia/masks/Viral Pneumonia-921.png,True,G20857,train,eb4849e88d0ac5ef1957f6e20b98e327,d47ca77aa20e6887
Viral Pneumonia-922.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-922.png,Viral Pneumonia/masks/Viral Pneumonia-922.png,True,G20858,train,c2d3af3ccbc15678b2266ac32b721b7b,9165c61a87cf3cc6
Viral Pneumonia-923.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-923.png,Viral Pneumonia/masks/Viral Pneumonia-923.png,True,G20859,val,aba1c954a270b33e30aa738233dd8f15,c137c61ac169b9cd
Viral Pneumonia-924.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-924.png,Viral Pneumonia/masks/Viral Pneumonia-924.png,True,G20860,test,e7860b6abdcf8607f2f3a2251e31c69a,d03fa66b99c36930
Viral Pneumonia-925.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-925.png,Viral Pneumonia/masks/Viral Pneumonia-925.png,True,G20861,train,60f419fd333d150d6cf9c8fe88906688,91f4c71e864bb8c3
Viral Pneumonia-926.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-926.png,Viral Pneumonia/masks/Viral Pneumonia-926.png,True,G20862,train,4a8099e636b352b60f5f107a1300f03d,d476c24bc51f8987
Viral Pneumonia-927.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-927.png,Viral Pneumonia/masks/Viral Pneumonia-927.png,True,G20863,val,a34ac874191a7af5004ae7c09f9c1da6,882aca5a69ef2d8b
Viral Pneumonia-928.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-928.png,Viral Pneumonia/masks/Viral Pneumonia-928.png,True,G20864,test,a352668ad44521cb451e233e7ccb9e4b,c0ba4f1e4c9a6d9c
Viral Pneumonia-929.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-929.png,Viral Pneumonia/masks/Viral Pneumonia-929.png,True,G20865,train,60404d9bd197871299611792245d48c8,813b3be5c74c8595
Viral Pneumonia-93.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-93.png,Viral Pneumonia/masks/Viral Pneumonia-93.png,True,G20866,train,903dd0757f892aa8c0c4479c7b8479cc,91e8ee7a7a591909
Viral Pneumonia-930.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-930.png,Viral Pneumonia/masks/Viral Pneumonia-930.png,True,G20867,train,7d814b4a97b23cb75c0893d3d93c4a51,912be33fb4c21a1e
Viral Pneumonia-931.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-931.png,Viral Pneumonia/masks/Viral Pneumonia-931.png,True,G20868,train,3e229457ee06e317a6211ed503873a70,d03d7b1e9a1c3ca4
Viral Pneumonia-932.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-932.png,Viral Pneumonia/masks/Viral Pneumonia-932.png,True,G20869,train,dbc8586899f9b3dd32840443fb7df100,d02b434a6a5d3ccf
Viral Pneumonia-933.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-933.png,Viral Pneumonia/masks/Viral Pneumonia-933.png,True,G20870,train,d380517e9b5a455f9cb24bcb8328b1fd,902bea5e5e5a9c86
Viral Pneumonia-934.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-934.png,Viral Pneumonia/masks/Viral Pneumonia-934.png,True,G20871,train,caf1bb461e41552d75ce0e4c9db4dc60,8a2bd59abe152c9c
Viral Pneumonia-935.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-935.png,Viral Pneumonia/masks/Viral Pneumonia-935.png,True,G20872,test,b9587bbbd0ac3a5960361ebcc11e4efd,903d2fa70a1f1f45
Viral Pneumonia-936.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-936.png,Viral Pneumonia/masks/Viral Pneumonia-936.png,True,G20873,train,b7f298a9e2297b76c25a9760dfbf987e,906f2f1e0d1d3c4e
Viral Pneumonia-937.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-937.png,Viral Pneumonia/masks/Viral Pneumonia-937.png,True,G20874,train,311609f949eb7956e54910ac491df7ef,90efc3165e4319d3
Viral Pneumonia-938.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-938.png,Viral Pneumonia/masks/Viral Pneumonia-938.png,True,G20875,train,62b9e0d37f836425cb82f43c26d174fa,d5ffe602185f5a10
Viral Pneumonia-939.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-939.png,Viral Pneumonia/masks/Viral Pneumonia-939.png,True,G20876,val,60267315452b8a6f1e95bb966a7bbf0f,9c64e21beb19e23c
Viral Pneumonia-94.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-94.png,Viral Pneumonia/masks/Viral Pneumonia-94.png,True,G20877,train,5bba48eb14aa2810bbd7916558f0c4be,907de63ea92f3a08
Viral Pneumonia-940.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-940.png,Viral Pneumonia/masks/Viral Pneumonia-940.png,True,G20878,train,c6378cf937df123b0a078ed12e0ffb3a,9129c71edaca39c9
Viral Pneumonia-941.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-941.png,Viral Pneumonia/masks/Viral Pneumonia-941.png,True,G20879,train,354fcfa5c724b6c0816203e12ec1625e,d46fa30b8b1b861e
Viral Pneumonia-942.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-942.png,Viral Pneumonia/masks/Viral Pneumonia-942.png,True,G20880,train,736a1663201aab8b8c623de7e8a6ca50,9126ea1aef3ba443
Viral Pneumonia-943.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-943.png,Viral Pneumonia/masks/Viral Pneumonia-943.png,True,G20881,train,9838de439bc4a0a09094180d3e0ea1a5,916dc75ac5192c4f
Viral Pneumonia-944.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-944.png,Viral Pneumonia/masks/Viral Pneumonia-944.png,True,G20882,train,d9bd72e4c89efc5681994fe703a6b5b5,d4e6c73bc70b814a
Viral Pneumonia-945.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-945.png,Viral Pneumonia/masks/Viral Pneumonia-945.png,True,G20883,train,c56c912c6771ec412d73f21aa4c36070,9065c71ac71ac27f
Viral Pneumonia-946.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-946.png,Viral Pneumonia/masks/Viral Pneumonia-946.png,True,G20884,train,4b28c08ff9176675f528be7baa9bf726,d464d31bc71fc303
Viral Pneumonia-947.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-947.png,Viral Pneumonia/masks/Viral Pneumonia-947.png,True,G20885,train,96208f6c4eabedfaaae723a1938240ab,d47ee71bc1c709c0
Viral Pneumonia-948.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-948.png,Viral Pneumonia/masks/Viral Pneumonia-948.png,True,G20886,train,2028543f677e9614e593e392d37379fd,9185e31ed2cbb8c9
Viral Pneumonia-949.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-949.png,Viral Pneumonia/masks/Viral Pneumonia-949.png,True,G20887,train,1e8d4249a2a63e43564119f699c34595,986eef1c471cc45c
Viral Pneumonia-95.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-95.png,Viral Pneumonia/masks/Viral Pneumonia-95.png,True,G20888,train,d3f63d8d130037c1d32cd7234d053c4d,c03de30fbc693ce0
Viral Pneumonia-950.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-950.png,Viral Pneumonia/masks/Viral Pneumonia-950.png,True,G20889,train,f60450b3ff6010288691bfec62600c35,9067c71eca19f549
Viral Pneumonia-951.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-951.png,Viral Pneumonia/masks/Viral Pneumonia-951.png,True,G20890,train,6d7d4474f65c70c3f7d8e117a5461cef,d42aeb1ec20f5a4b
Viral Pneumonia-952.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-952.png,Viral Pneumonia/masks/Viral Pneumonia-952.png,True,G20891,test,b6749b4fdba4174e1f1227420e015170,986ea73b9778e028
Viral Pneumonia-953.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-953.png,Viral Pneumonia/masks/Viral Pneumonia-953.png,True,G20892,val,f3ad695e63f644e5e1347a0b6209ff22,912bea3e5e4d1d48
Viral Pneumonia-954.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-954.png,Viral Pneumonia/masks/Viral Pneumonia-954.png,True,G20892,val,f3ad695e63f644e5e1347a0b6209ff22,912bea3e5e4d1d48
Viral Pneumonia-955.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-955.png,Viral Pneumonia/masks/Viral Pneumonia-955.png,True,G20893,val,264b8898d16973c8203d6b934d97453f,9038e16bbcc54f3c
Viral Pneumonia-956.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-956.png,Viral Pneumonia/masks/Viral Pneumonia-956.png,True,G20894,val,2d2bb5238d3b292f155e399eda5dc580,c422c30bfdcf38c6
Viral Pneumonia-957.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-957.png,Viral Pneumonia/masks/Viral Pneumonia-957.png,True,G20895,test,b914c647d7e18227adf483d5edcf7076,9521c23bed69346b
Viral Pneumonia-958.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-958.png,Viral Pneumonia/masks/Viral Pneumonia-958.png,True,G20896,test,1f32529ffafdd2c7b57b64bf5b811d62,85ab825fb600fdd2
Viral Pneumonia-959.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-959.png,Viral Pneumonia/masks/Viral Pneumonia-959.png,True,G20897,train,95d9ab2b16b7230e43c4b66e26225945,8c2aa35b834fd14f
Viral Pneumonia-96.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-96.png,Viral Pneumonia/masks/Viral Pneumonia-96.png,True,G20898,train,ba6d0a3398763fdfa2bad587f3db48e4,8125c61ee66abc6b
Viral Pneumonia-960.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-960.png,Viral Pneumonia/masks/Viral Pneumonia-960.png,True,G20899,train,fa9491c582ed4995dc17a3a2efb7b511,9165865a837ac57e
Viral Pneumonia-961.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-961.png,Viral Pneumonia/masks/Viral Pneumonia-961.png,True,G20900,train,79c8ba1f406a111b39c45260a58ba383,916fe61a453b544d
Viral Pneumonia-962.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-962.png,Viral Pneumonia/masks/Viral Pneumonia-962.png,True,G20901,train,f6e857d392263f3fb7adfe6a0543841b,906fe71ec11b3c45
Viral Pneumonia-963.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-963.png,Viral Pneumonia/masks/Viral Pneumonia-963.png,True,G20902,train,9919753c0e7a85f4622671349b8c95b2,d02b4e1a190b3faf
Viral Pneumonia-964.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-964.png,Viral Pneumonia/masks/Viral Pneumonia-964.png,True,G20903,train,2e377003bc77f1e91a45e41ca427d747,d02faf0b5d555c44
Viral Pneumonia-965.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-965.png,Viral Pneumonia/masks/Viral Pneumonia-965.png,True,G20904,test,863c63af43a6513f89f8228c4bf5ae9f,912be27af8795c41
Viral Pneumonia-966.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-966.png,Viral Pneumonia/masks/Viral Pneumonia-966.png,True,G20905,val,e8b71beda5154d7114c2fec9efa5b2fa,90f8e75ee1074a95
Viral Pneumonia-967.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-967.png,Viral Pneumonia/masks/Viral Pneumonia-967.png,True,G20906,val,863f7b64baf9f3988bede981ccb66f26,dd7ce41aa00f3aa5
Viral Pneumonia-968.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-968.png,Viral Pneumonia/masks/Viral Pneumonia-968.png,True,G20907,train,b725012a4bf870a715c78e06e28cb940,d42bc31ed51bd44a
Viral Pneumonia-969.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-969.png,Viral Pneumonia/masks/Viral Pneumonia-969.png,True,G20908,train,e62df262598e75f341d4d6a5810507ec,d46e071e4b4e68e6
Viral Pneumonia-97.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-97.png,Viral Pneumonia/masks/Viral Pneumonia-97.png,True,G20909,train,f5bed6c721a9b7e89c3093385779b212,913de71abec60a94
Viral Pneumonia-970.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-970.png,Viral Pneumonia/masks/Viral Pneumonia-970.png,True,G20910,test,2a547e7d108ee876b04797cd89680471,942ce31fe44e39c3
Viral Pneumonia-971.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-971.png,Viral Pneumonia/masks/Viral Pneumonia-971.png,True,G20911,train,0e8533f05fb54084f382100f21b20cb0,901be3473de1c23d
Viral Pneumonia-972.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-972.png,Viral Pneumonia/masks/Viral Pneumonia-972.png,True,G20912,train,ec68775e4b04d01d5f21fe09322a3153,946be31ad0613cf5
Viral Pneumonia-973.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-973.png,Viral Pneumonia/masks/Viral Pneumonia-973.png,True,G20913,train,cfa6151f580ec8bfbe748cab12a312eb,9054c71adbcd68e5
Viral Pneumonia-974.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-974.png,Viral Pneumonia/masks/Viral Pneumonia-974.png,True,G20914,train,dd777cc89e676b75f52c097ad9807b8e,d0e4c65ac739855e
Viral Pneumonia-975.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-975.png,Viral Pneumonia/masks/Viral Pneumonia-975.png,True,G20915,val,67aede6714b69e6594ff2ad3049bd131,d125c21bc66b9667
Viral Pneumonia-976.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-976.png,Viral Pneumonia/masks/Viral Pneumonia-976.png,True,G20916,test,53aada94db3a9991533f064c96c7ab3a,912bea3eda1a384b
Viral Pneumonia-977.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-977.png,Viral Pneumonia/masks/Viral Pneumonia-977.png,True,G20917,train,5c59ddd1e7956c90fd9b072341b15bd4,d4eae7a98119d14d
Viral Pneumonia-978.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-978.png,Viral Pneumonia/masks/Viral Pneumonia-978.png,True,G20918,train,4563c8c36ed4161d7a1065b254a9ff11,946c630fc71b931b
Viral Pneumonia-979.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-979.png,Viral Pneumonia/masks/Viral Pneumonia-979.png,True,G20919,train,bbd303046f45c6421fa6fa7b23676368,d56ce02fb4c20fb4
Viral Pneumonia-98.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-98.png,Viral Pneumonia/masks/Viral Pneumonia-98.png,True,G20920,train,623be48ea28f1c875af79eff8d757051,91b5c31eec4ab4c9
Viral Pneumonia-980.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-980.png,Viral Pneumonia/masks/Viral Pneumonia-980.png,True,G20921,train,800c0e8923715b37b8c47ae017f5772f,917d4e1dbac7a830
Viral Pneumonia-981.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-981.png,Viral Pneumonia/masks/Viral Pneumonia-981.png,True,G20922,train,0e7523dd99034a8da7d6f45b0a7fdcc0,9939e14f3cc04bb5
Viral Pneumonia-982.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-982.png,Viral Pneumonia/masks/Viral Pneumonia-982.png,True,G20923,train,9d3893eb20389d484251ffef02fa0b3d,9939c51f94461e57
Viral Pneumonia-983.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-983.png,Viral Pneumonia/masks/Viral Pneumonia-983.png,True,G20924,train,62bbc965bf8c8fc6d640b1f6ae98ddbd,d422c31bc71f8ce6
Viral Pneumonia-984.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-984.png,Viral Pneumonia/masks/Viral Pneumonia-984.png,True,G20925,train,9aec9692bc25d64c6caa711103ab263b,94a9e71bc61bc44b
Viral Pneumonia-985.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-985.png,Viral Pneumonia/masks/Viral Pneumonia-985.png,True,G20926,train,2dc69d781a0d072ac5b71b4636764ce7,902eee38e12e94cf
Viral Pneumonia-986.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-986.png,Viral Pneumonia/masks/Viral Pneumonia-986.png,True,G20927,train,ce1fea34442492b85f05cae5347f6061,c02aa257a25dafd5
Viral Pneumonia-987.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-987.png,Viral Pneumonia/masks/Viral Pneumonia-987.png,True,G20928,train,3c4c2babd5732e926790c5a8beafd91d,d42af71fe02a1655
Viral Pneumonia-988.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-988.png,Viral Pneumonia/masks/Viral Pneumonia-988.png,True,G20929,val,5f76b3c2b79c73027ffd5e4151fe49c0,8837e218c33bb537
Viral Pneumonia-989.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-989.png,Viral Pneumonia/masks/Viral Pneumonia-989.png,True,G20930,train,629606c6993e91449066ab7e530a220e,8865e67a877aa316
Viral Pneumonia-99.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-99.png,Viral Pneumonia/masks/Viral Pneumonia-99.png,True,G20931,train,78e6e601fc379375b3107bf5a9d35c41,d46cc33bcb4b984a
Viral Pneumonia-990.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-990.png,Viral Pneumonia/masks/Viral Pneumonia-990.png,True,G20932,val,80ca3f169cc373939d7c5012ff17215c,cc22e31fd12db04f
Viral Pneumonia-991.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-991.png,Viral Pneumonia/masks/Viral Pneumonia-991.png,True,G20933,val,5b52cd12a768990422dff0a07b0c8580,d04eab4ead7aa446
Viral Pneumonia-992.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-992.png,Viral Pneumonia/masks/Viral Pneumonia-992.png,True,G20934,test,7a0b1d4e2adeb3d7821d675b4b14b6d5,d403e70ba7cb2678
Viral Pneumonia-993.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-993.png,Viral Pneumonia/masks/Viral Pneumonia-993.png,True,G20935,test,36e0fb621fdd3a6936a93c06d3ee9f8c,9531e31aee4b3c38
Viral Pneumonia-994.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-994.png,Viral Pneumonia/masks/Viral Pneumonia-994.png,True,G20936,test,d06a56f63ec73c0b45e3688fc7d9a174,956e0f1815594d7d
Viral Pneumonia-995.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-995.png,Viral Pneumonia/masks/Viral Pneumonia-995.png,True,G20937,train,3b87d27355f958edbb3ab2f79834ec10,90a1c71fd64bbc4a
Viral Pneumonia-996.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-996.png,Viral Pneumonia/masks/Viral Pneumonia-996.png,True,G20938,train,f5b2a4a3a3ff4efca48cb41f9ef74ce3,9061e67ec23f412f
Viral Pneumonia-997.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-997.png,Viral Pneumonia/masks/Viral Pneumonia-997.png,True,G20939,train,06057f9a26f657073f446e0650668d5c,8c2d821b831b577f
Viral Pneumonia-998.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-998.png,Viral Pneumonia/masks/Viral Pneumonia-998.png,True,G20940,train,8dc33fbac06668f6ef48e1d7ee0227f2,d47ce6aa1aa0875e
Viral Pneumonia-999.png,Viral Pneumonia,Viral Pneumonia/images/Viral Pneumonia-999.png,Viral Pneumonia/masks/Viral Pneumonia-999.png,True,G20941,train,385992c8e0971305a95a8dc9880b1e5e,d02fd7495a481d9d
''')

with open("splits/manifest.csv", "w", encoding="utf-8") as f:
    f.write('''image_id,label,image_path,rel_image_path,mask_path,rel_mask_path,has_mask,mask_match_size,image_width,image_height,color_mode,mean_brightness,file_hash
COVID-1.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1.png,COVID/images/COVID-1.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1.png,COVID/masks/COVID-1.png,True,False,299,299,L,145.9,f562f6a6290c0a8e27afc9f994625380
COVID-10.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-10.png,COVID/images/COVID-10.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-10.png,COVID/masks/COVID-10.png,True,False,299,299,L,141.36,f29158352a454f1d39ed9924406f99ec
COVID-100.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-100.png,COVID/images/COVID-100.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-100.png,COVID/masks/COVID-100.png,True,False,299,299,L,152.88,6da68dc77f90d6ed08fc41bf3c3d3cae
COVID-1000.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1000.png,COVID/images/COVID-1000.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1000.png,COVID/masks/COVID-1000.png,True,False,299,299,L,162.77,641cebe2e5e95cfcd94a6dc2b5d1b3ee
COVID-1001.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1001.png,COVID/images/COVID-1001.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1001.png,COVID/masks/COVID-1001.png,True,False,299,299,L,129.89,b34c634b7eb9d9fd6b801b5218d2e684
COVID-1002.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1002.png,COVID/images/COVID-1002.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1002.png,COVID/masks/COVID-1002.png,True,False,299,299,L,175.74,422f0565e6f9170471119772d1c04908
COVID-1003.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1003.png,COVID/images/COVID-1003.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1003.png,COVID/masks/COVID-1003.png,True,False,299,299,L,160.76,41550c0344c667f07122a3d8d766f9dc
COVID-1004.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1004.png,COVID/images/COVID-1004.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1004.png,COVID/masks/COVID-1004.png,True,False,299,299,L,135.65,20b6575c31fc2a456335aeab1ba6e034
COVID-1005.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1005.png,COVID/images/COVID-1005.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1005.png,COVID/masks/COVID-1005.png,True,False,299,299,L,140.92,711e4fb3cb56e2782c92375b29361621
COVID-1006.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1006.png,COVID/images/COVID-1006.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1006.png,COVID/masks/COVID-1006.png,True,False,299,299,L,147.76,2a2482e5c9d19e819c2890f53abbca07
COVID-1007.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1007.png,COVID/images/COVID-1007.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1007.png,COVID/masks/COVID-1007.png,True,False,299,299,L,127.01,b437244040aaf8ece7c3acd778011715
COVID-1008.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1008.png,COVID/images/COVID-1008.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1008.png,COVID/masks/COVID-1008.png,True,False,299,299,L,135.13,d943b445e77b1322989f67980e4c3442
COVID-1009.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1009.png,COVID/images/COVID-1009.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1009.png,COVID/masks/COVID-1009.png,True,False,299,299,L,136.42,f29444e7785c1c5e855bd8f14100d579
COVID-101.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-101.png,COVID/images/COVID-101.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-101.png,COVID/masks/COVID-101.png,True,False,299,299,L,117.82,9863c004407e2c99e62c9502dd1848b4
COVID-1010.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1010.png,COVID/images/COVID-1010.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1010.png,COVID/masks/COVID-1010.png,True,False,299,299,L,140.38,eda7c8d33a7b248b5aed2ffe7da9f0cc
COVID-1011.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1011.png,COVID/images/COVID-1011.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1011.png,COVID/masks/COVID-1011.png,True,False,299,299,L,153.92,1f6cea7496fed4e1aa8f93f7eb9ae7c5
COVID-1012.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1012.png,COVID/images/COVID-1012.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1012.png,COVID/masks/COVID-1012.png,True,False,299,299,L,152.68,3894c52f655531a9a903c27f3d7e225d
COVID-1013.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1013.png,COVID/images/COVID-1013.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1013.png,COVID/masks/COVID-1013.png,True,False,299,299,L,126.34,64a5bbe062c9ccd69a3b854d92f92ca8
COVID-1014.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1014.png,COVID/images/COVID-1014.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1014.png,COVID/masks/COVID-1014.png,True,False,299,299,L,127.95,a5ac932c1215dcb6261aa07f58fb9e0a
COVID-1015.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1015.png,COVID/images/COVID-1015.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1015.png,COVID/masks/COVID-1015.png,True,False,299,299,L,108.47,9978b12e04aa3c12de5d9a28f20a83af
COVID-1016.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1016.png,COVID/images/COVID-1016.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1016.png,COVID/masks/COVID-1016.png,True,False,299,299,L,139.04,f2d56e2515cc1f3ec30b9c3c4cda9953
COVID-1017.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1017.png,COVID/images/COVID-1017.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1017.png,COVID/masks/COVID-1017.png,True,False,299,299,L,152.51,ae32ccc653e4e84308d8e05a810dc7e2
COVID-1018.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1018.png,COVID/images/COVID-1018.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1018.png,COVID/masks/COVID-1018.png,True,False,299,299,L,140.58,f441ea861f677529d1c06fe6345c50f6
COVID-1019.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1019.png,COVID/images/COVID-1019.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1019.png,COVID/masks/COVID-1019.png,True,False,299,299,L,139.68,0440accd4567c96ffbc439b22a07005e
COVID-102.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-102.png,COVID/images/COVID-102.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-102.png,COVID/masks/COVID-102.png,True,False,299,299,L,97.79,da18441924afa55afeb2e320e6504e29
COVID-1020.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1020.png,COVID/images/COVID-1020.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1020.png,COVID/masks/COVID-1020.png,True,False,299,299,L,141.94,0b7a97c505f1f8250be9e0914221ec9a
COVID-1021.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1021.png,COVID/images/COVID-1021.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1021.png,COVID/masks/COVID-1021.png,True,False,299,299,L,184.96,3d016c3a3d079a00f6bb190299644a9d
COVID-1022.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1022.png,COVID/images/COVID-1022.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1022.png,COVID/masks/COVID-1022.png,True,False,299,299,L,144.48,f364100e0cfd1dd0c2ce20fd03cf1527
COVID-1023.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1023.png,COVID/images/COVID-1023.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1023.png,COVID/masks/COVID-1023.png,True,False,299,299,L,143.75,5f5bbb6f85ad26de1d4f03b3763660b9
COVID-1024.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1024.png,COVID/images/COVID-1024.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1024.png,COVID/masks/COVID-1024.png,True,False,299,299,L,131.55,1b1d40da3e648452a4c6cf172accd588
COVID-1025.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1025.png,COVID/images/COVID-1025.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1025.png,COVID/masks/COVID-1025.png,True,False,299,299,L,147.08,1e5199f2c06c5c131309f8d2e67ed63e
COVID-1026.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1026.png,COVID/images/COVID-1026.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1026.png,COVID/masks/COVID-1026.png,True,False,299,299,L,150.02,fc02f8a1721f67c94c65f1f9c0c1d118
COVID-1027.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1027.png,COVID/images/COVID-1027.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1027.png,COVID/masks/COVID-1027.png,True,False,299,299,L,166.85,5ece33729cf3c03f30b8d86f11fb80db
COVID-1028.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1028.png,COVID/images/COVID-1028.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1028.png,COVID/masks/COVID-1028.png,True,False,299,299,L,160.09,0ffd904f05cb38ebeb74d40977cf88ae
COVID-1029.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1029.png,COVID/images/COVID-1029.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1029.png,COVID/masks/COVID-1029.png,True,False,299,299,L,146.28,3b34760295806af4ab977a1c73460035
COVID-103.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-103.png,COVID/images/COVID-103.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-103.png,COVID/masks/COVID-103.png,True,False,299,299,L,143.27,4f054f061e219031d3abbd9b7db11303
COVID-1030.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1030.png,COVID/images/COVID-1030.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1030.png,COVID/masks/COVID-1030.png,True,False,299,299,L,146.24,ab90fc6e323d159d203de0e5e8bb7806
COVID-1031.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1031.png,COVID/images/COVID-1031.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1031.png,COVID/masks/COVID-1031.png,True,False,299,299,L,149.47,0585d2567cb15521a2160828926f7ed3
COVID-1032.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1032.png,COVID/images/COVID-1032.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1032.png,COVID/masks/COVID-1032.png,True,False,299,299,L,125.13,6e50783d746241708e12f0d02d644f88
COVID-1033.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1033.png,COVID/images/COVID-1033.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1033.png,COVID/masks/COVID-1033.png,True,False,299,299,L,165.02,99638756f0f3ca8866372046319610ac
COVID-1034.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1034.png,COVID/images/COVID-1034.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1034.png,COVID/masks/COVID-1034.png,True,False,299,299,L,147.44,7003b2fb37b33eff77c666236dd8dfc7
COVID-1035.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1035.png,COVID/images/COVID-1035.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1035.png,COVID/masks/COVID-1035.png,True,False,299,299,L,110.18,c85c8d75cf5673cefeec3ee2121e601a
COVID-1036.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1036.png,COVID/images/COVID-1036.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1036.png,COVID/masks/COVID-1036.png,True,False,299,299,L,151.87,8f6d679cb91c92da07706ae6e7c566f2
COVID-1037.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1037.png,COVID/images/COVID-1037.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1037.png,COVID/masks/COVID-1037.png,True,False,299,299,L,143.52,5f5b53a6d93f76714bd556bf786e6a11
COVID-1038.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1038.png,COVID/images/COVID-1038.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1038.png,COVID/masks/COVID-1038.png,True,False,299,299,L,122.86,8489cc060ca2a30cc3703b9809e4e5d3
COVID-1039.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1039.png,COVID/images/COVID-1039.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1039.png,COVID/masks/COVID-1039.png,True,False,299,299,L,154.04,e28bd145f0525a276eeb87c11e754120
COVID-104.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-104.png,COVID/images/COVID-104.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-104.png,COVID/masks/COVID-104.png,True,False,299,299,L,138.74,aeb7412ef18fdd0b092aae10a1cb6e8d
COVID-1040.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1040.png,COVID/images/COVID-1040.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1040.png,COVID/masks/COVID-1040.png,True,False,299,299,L,160.1,ecaa72b0546e97bb9f59127d4e40315d
COVID-1041.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1041.png,COVID/images/COVID-1041.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1041.png,COVID/masks/COVID-1041.png,True,False,299,299,L,150.38,f504d7210018a5a350a968246f5536c0
COVID-1042.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1042.png,COVID/images/COVID-1042.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1042.png,COVID/masks/COVID-1042.png,True,False,299,299,L,147.78,8eeab2985a667f6fb11442323cf6384f
COVID-1043.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1043.png,COVID/images/COVID-1043.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1043.png,COVID/masks/COVID-1043.png,True,False,299,299,L,154.59,57fe93f9a07c4e6b0254eb3a3cffc60e
COVID-1044.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1044.png,COVID/images/COVID-1044.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1044.png,COVID/masks/COVID-1044.png,True,False,299,299,L,159.43,03bde6766e6a5429d9c792683829fe80
COVID-1045.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1045.png,COVID/images/COVID-1045.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1045.png,COVID/masks/COVID-1045.png,True,False,299,299,L,134.01,c774f3e94c420870d5937a26ede1379f
COVID-1046.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1046.png,COVID/images/COVID-1046.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1046.png,COVID/masks/COVID-1046.png,True,False,299,299,L,189.66,fddafead84db457677f26c59afb89853
COVID-1047.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1047.png,COVID/images/COVID-1047.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1047.png,COVID/masks/COVID-1047.png,True,False,299,299,L,128.31,b8fa4d8a7373c5d19a9d591a4e01a514
COVID-1048.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1048.png,COVID/images/COVID-1048.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1048.png,COVID/masks/COVID-1048.png,True,False,299,299,L,146.57,fc5a9c87f6bbc6a7a82f2e16a669831f
COVID-1049.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1049.png,COVID/images/COVID-1049.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1049.png,COVID/masks/COVID-1049.png,True,False,299,299,L,190.78,d475d9b379311355989defa0573d6a3d
COVID-105.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-105.png,COVID/images/COVID-105.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-105.png,COVID/masks/COVID-105.png,True,False,299,299,L,110.62,2f71ca55a35afee7c7107406943f8e64
COVID-1050.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1050.png,COVID/images/COVID-1050.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1050.png,COVID/masks/COVID-1050.png,True,False,299,299,L,167.41,69de877f6a11a365249888efb4b95ccf
COVID-1051.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1051.png,COVID/images/COVID-1051.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1051.png,COVID/masks/COVID-1051.png,True,False,299,299,L,135.44,c532b87701be2044072e45feb4cb2c67
COVID-1052.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1052.png,COVID/images/COVID-1052.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1052.png,COVID/masks/COVID-1052.png,True,False,299,299,L,115.17,5825db2e8b174c3aebed918c8192d62d
COVID-1053.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1053.png,COVID/images/COVID-1053.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1053.png,COVID/masks/COVID-1053.png,True,False,299,299,L,160.85,23828496f8808117cc852892543afa52
COVID-1054.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1054.png,COVID/images/COVID-1054.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1054.png,COVID/masks/COVID-1054.png,True,False,299,299,L,126.97,60bcda139331ef771fbce0973c6180f5
COVID-1055.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1055.png,COVID/images/COVID-1055.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1055.png,COVID/masks/COVID-1055.png,True,False,299,299,L,155.31,574bcad72ebf059355378a4d9c594016
COVID-1056.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1056.png,COVID/images/COVID-1056.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1056.png,COVID/masks/COVID-1056.png,True,False,299,299,L,107.98,4777d27dbc9a70eb816ae262cbeb8ee6
COVID-1057.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1057.png,COVID/images/COVID-1057.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1057.png,COVID/masks/COVID-1057.png,True,False,299,299,L,69.09,5da69b241b0df94695cb13b81c146560
COVID-1058.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1058.png,COVID/images/COVID-1058.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1058.png,COVID/masks/COVID-1058.png,True,False,299,299,L,151.82,5a258309e2db70fa5efcbd682a5874a2
COVID-1059.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1059.png,COVID/images/COVID-1059.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1059.png,COVID/masks/COVID-1059.png,True,False,299,299,L,143.91,d31e0d0174e85185e73fec436db22901
COVID-106.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-106.png,COVID/images/COVID-106.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-106.png,COVID/masks/COVID-106.png,True,False,299,299,L,166.97,112d8695e04360a6ae356a95b6ab2214
COVID-1060.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1060.png,COVID/images/COVID-1060.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1060.png,COVID/masks/COVID-1060.png,True,False,299,299,L,67.41,83f0c050d1079c87c58cf1913308e498
COVID-1061.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1061.png,COVID/images/COVID-1061.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1061.png,COVID/masks/COVID-1061.png,True,False,299,299,L,150.22,4d8844b1e01521ebb56668a3620dd606
COVID-1062.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1062.png,COVID/images/COVID-1062.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1062.png,COVID/masks/COVID-1062.png,True,False,299,299,L,145.54,5f8301dfc1fdd0bb4f59cfd43e87bbe2
COVID-1063.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1063.png,COVID/images/COVID-1063.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1063.png,COVID/masks/COVID-1063.png,True,False,299,299,L,60.63,8bcbff2e3dc9b3b870e1f1b185b8b84a
COVID-1064.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1064.png,COVID/images/COVID-1064.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1064.png,COVID/masks/COVID-1064.png,True,False,299,299,L,149.94,868c8b02a17fa6e9bf201e4e880d91ff
COVID-1065.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1065.png,COVID/images/COVID-1065.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1065.png,COVID/masks/COVID-1065.png,True,False,299,299,L,164.74,86b1e3561dd37524d8807408350fff89
COVID-1066.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1066.png,COVID/images/COVID-1066.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1066.png,COVID/masks/COVID-1066.png,True,False,299,299,L,149.31,76037e4b3fdce076d97b9c14f90dfb0b
COVID-1067.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1067.png,COVID/images/COVID-1067.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1067.png,COVID/masks/COVID-1067.png,True,False,299,299,L,146.36,1ed4803fe8c8f810b81c4f087a35c89b
COVID-1068.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1068.png,COVID/images/COVID-1068.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1068.png,COVID/masks/COVID-1068.png,True,False,299,299,L,161.92,d893c11f9211c5e8eb9433f6ff82c32b
COVID-1069.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1069.png,COVID/images/COVID-1069.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1069.png,COVID/masks/COVID-1069.png,True,False,299,299,L,151.57,1b772c5e32dac333535ad0bba6080868
COVID-107.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-107.png,COVID/images/COVID-107.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-107.png,COVID/masks/COVID-107.png,True,False,299,299,L,135.84,db08feee7df0fc3bccfdc51be848f7fa
COVID-1070.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1070.png,COVID/images/COVID-1070.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1070.png,COVID/masks/COVID-1070.png,True,False,299,299,L,145.61,e3462978bc7844c3b6e2e339512fc231
COVID-1071.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1071.png,COVID/images/COVID-1071.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1071.png,COVID/masks/COVID-1071.png,True,False,299,299,L,150.94,031d8f26ccc394e326d64f91c11d0ee1
COVID-1072.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1072.png,COVID/images/COVID-1072.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1072.png,COVID/masks/COVID-1072.png,True,False,299,299,L,153.98,cc325ceed9c656f7a4b7f9da5d42f1f7
COVID-1073.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1073.png,COVID/images/COVID-1073.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1073.png,COVID/masks/COVID-1073.png,True,False,299,299,L,144.32,2a90b2906ac09e244f1a2a1b6a7d911b
COVID-1074.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1074.png,COVID/images/COVID-1074.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1074.png,COVID/masks/COVID-1074.png,True,False,299,299,L,156.9,fb6cf32849fbd8f84a5edbdd99cb924c
COVID-1075.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1075.png,COVID/images/COVID-1075.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1075.png,COVID/masks/COVID-1075.png,True,False,299,299,L,139.37,c7a1d6838363abf5fe993a9f7f79eda6
COVID-1076.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1076.png,COVID/images/COVID-1076.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1076.png,COVID/masks/COVID-1076.png,True,False,299,299,L,141.96,8a801ac4df1ed73d85063599bb9e5fbd
COVID-1077.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1077.png,COVID/images/COVID-1077.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1077.png,COVID/masks/COVID-1077.png,True,False,299,299,L,157.37,3a0286ca32f42b486da258e151b44380
COVID-1078.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1078.png,COVID/images/COVID-1078.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1078.png,COVID/masks/COVID-1078.png,True,False,299,299,L,124.27,6f6b202143ddcdc1d1a6847314750b07
COVID-1079.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1079.png,COVID/images/COVID-1079.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1079.png,COVID/masks/COVID-1079.png,True,False,299,299,L,130.39,10da2f32915318886e6e99736e36345f
COVID-108.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-108.png,COVID/images/COVID-108.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-108.png,COVID/masks/COVID-108.png,True,False,299,299,L,126.82,11541d02fbbbf129ff00646457cc7894
COVID-1080.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1080.png,COVID/images/COVID-1080.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1080.png,COVID/masks/COVID-1080.png,True,False,299,299,L,133.9,b5972c74865ae6000db53bd38ae3c6ca
COVID-1081.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1081.png,COVID/images/COVID-1081.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1081.png,COVID/masks/COVID-1081.png,True,False,299,299,L,85.67,40ad0a419f1831adbf60b9a34e002de6
COVID-1082.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1082.png,COVID/images/COVID-1082.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1082.png,COVID/masks/COVID-1082.png,True,False,299,299,L,121.6,b33c65ed8b6880f8b22365704d4afced
COVID-1083.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1083.png,COVID/images/COVID-1083.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1083.png,COVID/masks/COVID-1083.png,True,False,299,299,L,160.86,ab5e260348cef86843c3664fa36bc466
COVID-1084.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1084.png,COVID/images/COVID-1084.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1084.png,COVID/masks/COVID-1084.png,True,False,299,299,L,137.99,7e31361ecad80a70bb3c928c7f1381ee
COVID-1085.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1085.png,COVID/images/COVID-1085.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1085.png,COVID/masks/COVID-1085.png,True,False,299,299,L,144.51,7fe8510625d4c75fdeeac9722b2272cb
COVID-1086.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1086.png,COVID/images/COVID-1086.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1086.png,COVID/masks/COVID-1086.png,True,False,299,299,L,57.84,ff0984129ca85de36ec20863ca7b243d
COVID-1087.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1087.png,COVID/images/COVID-1087.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1087.png,COVID/masks/COVID-1087.png,True,False,299,299,L,151.64,24e079e47f2ff3f6b8d047fefb23c414
COVID-1088.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1088.png,COVID/images/COVID-1088.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1088.png,COVID/masks/COVID-1088.png,True,False,299,299,L,123.03,09b276ab39ce7598ab40440f0881e1e3
COVID-1089.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1089.png,COVID/images/COVID-1089.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1089.png,COVID/masks/COVID-1089.png,True,False,299,299,L,140.25,b92b2ab2c9d014cdcd510895705d3c8f
COVID-109.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-109.png,COVID/images/COVID-109.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-109.png,COVID/masks/COVID-109.png,True,False,299,299,L,119.51,0bc92192ffd6b2f7299ba8774c3196ea
COVID-1090.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1090.png,COVID/images/COVID-1090.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1090.png,COVID/masks/COVID-1090.png,True,False,299,299,L,114.08,0dbd95f8d23d7e307d39f21465079281
COVID-1091.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1091.png,COVID/images/COVID-1091.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1091.png,COVID/masks/COVID-1091.png,True,False,299,299,L,173.39,85105e98d54408af74da55fbd986c737
COVID-1092.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1092.png,COVID/images/COVID-1092.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1092.png,COVID/masks/COVID-1092.png,True,False,299,299,L,164.16,cae9f22eae7d0f7b7f39529db8eac529
COVID-1093.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1093.png,COVID/images/COVID-1093.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1093.png,COVID/masks/COVID-1093.png,True,False,299,299,L,156.23,7f60ae0e111f6e58869ae10ece573f08
COVID-1094.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1094.png,COVID/images/COVID-1094.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1094.png,COVID/masks/COVID-1094.png,True,False,299,299,L,182.58,10e847e143522143239ae2d884e700f2
COVID-1095.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1095.png,COVID/images/COVID-1095.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1095.png,COVID/masks/COVID-1095.png,True,False,299,299,L,104.81,be6f59bac6445abba965e1d66c21ee1a
COVID-1096.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1096.png,COVID/images/COVID-1096.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1096.png,COVID/masks/COVID-1096.png,True,False,299,299,L,68.83,2646d7593575d43a7a8535ac54149da9
COVID-1097.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1097.png,COVID/images/COVID-1097.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1097.png,COVID/masks/COVID-1097.png,True,False,299,299,L,172.58,bee49db3088c8b2c6cf8bca4b45eede1
COVID-1098.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1098.png,COVID/images/COVID-1098.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1098.png,COVID/masks/COVID-1098.png,True,False,299,299,L,117.21,30aee2e61b2f460fde11f32c436389d8
COVID-1099.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1099.png,COVID/images/COVID-1099.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1099.png,COVID/masks/COVID-1099.png,True,False,299,299,L,155.97,a6573f9c1ec18a000fc24480e9367849
COVID-11.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-11.png,COVID/images/COVID-11.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-11.png,COVID/masks/COVID-11.png,True,False,299,299,L,149.83,f487c8a6116b7099336edd0d52821baa
COVID-110.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-110.png,COVID/images/COVID-110.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-110.png,COVID/masks/COVID-110.png,True,False,299,299,L,147.29,9b1c4cd3ceaf30ae1dc3db619f332258
COVID-1100.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1100.png,COVID/images/COVID-1100.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1100.png,COVID/masks/COVID-1100.png,True,False,299,299,L,164.69,57bc5ea485e7dee7c1ccb71673f11e79
COVID-1101.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1101.png,COVID/images/COVID-1101.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1101.png,COVID/masks/COVID-1101.png,True,False,299,299,L,117.74,74df7dc24241d552b784a03b33158a6c
COVID-1102.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1102.png,COVID/images/COVID-1102.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1102.png,COVID/masks/COVID-1102.png,True,False,299,299,L,162.62,a652a3ecc9b1e64ba43ef0b1953ebe92
COVID-1103.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1103.png,COVID/images/COVID-1103.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1103.png,COVID/masks/COVID-1103.png,True,False,299,299,L,147.08,8fa9de678fc2e1da46fe86c002d0c57f
COVID-1104.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1104.png,COVID/images/COVID-1104.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1104.png,COVID/masks/COVID-1104.png,True,False,299,299,L,130.19,15585406d33f1abb877705b58c8d335e
COVID-1105.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1105.png,COVID/images/COVID-1105.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1105.png,COVID/masks/COVID-1105.png,True,False,299,299,L,145.88,ac0579b9b2fade3405d0b06a8f1b085a
COVID-1106.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1106.png,COVID/images/COVID-1106.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1106.png,COVID/masks/COVID-1106.png,True,False,299,299,L,172.84,ae1f5192a1b932d85cde01b99fa250b8
COVID-1107.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1107.png,COVID/images/COVID-1107.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1107.png,COVID/masks/COVID-1107.png,True,False,299,299,L,129.35,cc0367594b61650ee5939d45ed72388b
COVID-1108.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1108.png,COVID/images/COVID-1108.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1108.png,COVID/masks/COVID-1108.png,True,False,299,299,L,159.88,1d1a43ed6bdc0f833a33a3f93276c7ac
COVID-1109.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1109.png,COVID/images/COVID-1109.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1109.png,COVID/masks/COVID-1109.png,True,False,299,299,L,163.87,21afe8c77a731bcd72a4a230c96edbf4
COVID-111.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-111.png,COVID/images/COVID-111.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-111.png,COVID/masks/COVID-111.png,True,False,299,299,L,131.47,5f21a41c8538df777d8e536f7ad21ca2
COVID-1110.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1110.png,COVID/images/COVID-1110.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1110.png,COVID/masks/COVID-1110.png,True,False,299,299,L,144.05,964f51e136ec692cf3e2ed0060ae05c5
COVID-1111.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1111.png,COVID/images/COVID-1111.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1111.png,COVID/masks/COVID-1111.png,True,False,299,299,L,154.72,8ee941404ac3e5f937b3d4b11ddd4a36
COVID-1112.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1112.png,COVID/images/COVID-1112.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1112.png,COVID/masks/COVID-1112.png,True,False,299,299,L,133.08,d52675182af7fb7ed518b7b50638252d
COVID-1113.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1113.png,COVID/images/COVID-1113.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1113.png,COVID/masks/COVID-1113.png,True,False,299,299,L,112.94,bf3695d53d6e58f47b063234332bc788
COVID-1114.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1114.png,COVID/images/COVID-1114.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1114.png,COVID/masks/COVID-1114.png,True,False,299,299,L,160.94,8643b3fca5be14097ccccbd9ffb5de78
COVID-1115.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1115.png,COVID/images/COVID-1115.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1115.png,COVID/masks/COVID-1115.png,True,False,299,299,L,151.85,62cfe387a4650bfc8bb0550bf2edd431
COVID-1116.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1116.png,COVID/images/COVID-1116.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1116.png,COVID/masks/COVID-1116.png,True,False,299,299,L,134.87,5e198b4f2bc50b3c0d87ab0dbbed3f83
COVID-1117.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1117.png,COVID/images/COVID-1117.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1117.png,COVID/masks/COVID-1117.png,True,False,299,299,L,66.2,5cb954c5009a641988973328333e2ffa
COVID-1118.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1118.png,COVID/images/COVID-1118.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1118.png,COVID/masks/COVID-1118.png,True,False,299,299,L,159.64,f3110a82f2ad0c6df0187423b713355a
COVID-1119.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1119.png,COVID/images/COVID-1119.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1119.png,COVID/masks/COVID-1119.png,True,False,299,299,L,142.7,b8d04969750227114fc292e326b9e118
COVID-112.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-112.png,COVID/images/COVID-112.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-112.png,COVID/masks/COVID-112.png,True,False,299,299,L,128.53,b66db83b183446e4153e1972f9992b44
COVID-1120.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1120.png,COVID/images/COVID-1120.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1120.png,COVID/masks/COVID-1120.png,True,False,299,299,L,57.68,c93bf82d4e6a4efe040cc31cdd17f086
COVID-1121.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1121.png,COVID/images/COVID-1121.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1121.png,COVID/masks/COVID-1121.png,True,False,299,299,L,171.52,2e8f30bb33e011ca2acd4a4ec36553c7
COVID-1122.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1122.png,COVID/images/COVID-1122.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1122.png,COVID/masks/COVID-1122.png,True,False,299,299,L,131.48,35a6c7fd5f9c150dd04b921830d73ad1
COVID-1123.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1123.png,COVID/images/COVID-1123.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1123.png,COVID/masks/COVID-1123.png,True,False,299,299,L,164.73,4c4a7ae5607059587d0912057f5d3f58
COVID-1124.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1124.png,COVID/images/COVID-1124.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1124.png,COVID/masks/COVID-1124.png,True,False,299,299,L,179.49,1a75d350c73a0c04813bc58f85c1fa22
COVID-1125.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1125.png,COVID/images/COVID-1125.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1125.png,COVID/masks/COVID-1125.png,True,False,299,299,L,141.66,b7e40a76e680e8b7800595210a2c1e64
COVID-1126.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1126.png,COVID/images/COVID-1126.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1126.png,COVID/masks/COVID-1126.png,True,False,299,299,L,171.94,33278551d9c87843f1a6d79857ee20c3
COVID-1127.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1127.png,COVID/images/COVID-1127.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1127.png,COVID/masks/COVID-1127.png,True,False,299,299,L,145.02,9b8428237765159299d50dd28063b9d2
COVID-1128.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1128.png,COVID/images/COVID-1128.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1128.png,COVID/masks/COVID-1128.png,True,False,299,299,L,168.59,a0fdbd95c2a91048a9b710727bfd29bc
COVID-1129.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1129.png,COVID/images/COVID-1129.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1129.png,COVID/masks/COVID-1129.png,True,False,299,299,L,107.76,6743ef034572faa373593518b09187ac
COVID-113.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-113.png,COVID/images/COVID-113.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-113.png,COVID/masks/COVID-113.png,True,False,299,299,L,151.2,5b6b3fe6c1c0f0df61795c2f87d579d1
COVID-1130.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1130.png,COVID/images/COVID-1130.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1130.png,COVID/masks/COVID-1130.png,True,False,299,299,L,113.65,c9156c835d837ae58b0d5fc82a33f7e4
COVID-1131.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1131.png,COVID/images/COVID-1131.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1131.png,COVID/masks/COVID-1131.png,True,False,299,299,L,165.95,47d825580327bfa578fc54a8cbf4a475
COVID-1132.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1132.png,COVID/images/COVID-1132.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1132.png,COVID/masks/COVID-1132.png,True,False,299,299,L,136.81,68e469b1220f8b0e9b3d81b843272db5
COVID-1133.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1133.png,COVID/images/COVID-1133.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1133.png,COVID/masks/COVID-1133.png,True,False,299,299,L,150.07,dd7623613c77eb779dc69ece6fbec698
COVID-1134.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1134.png,COVID/images/COVID-1134.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1134.png,COVID/masks/COVID-1134.png,True,False,299,299,L,119.09,feb23c78146a86a0333b0eccf1d80f5d
COVID-1135.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1135.png,COVID/images/COVID-1135.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1135.png,COVID/masks/COVID-1135.png,True,False,299,299,L,117.43,e308de36726b904d2a6ed188c4323d34
COVID-1136.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1136.png,COVID/images/COVID-1136.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1136.png,COVID/masks/COVID-1136.png,True,False,299,299,L,150.02,31ccf8c346845f4fb0c60cb7b9192f59
COVID-1137.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1137.png,COVID/images/COVID-1137.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1137.png,COVID/masks/COVID-1137.png,True,False,299,299,L,138.17,df738687873ebe522bce53e2a3231c4b
COVID-1138.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1138.png,COVID/images/COVID-1138.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1138.png,COVID/masks/COVID-1138.png,True,False,299,299,L,151.96,2cbcaa114ba3c154c99654a0b5e08bce
COVID-1139.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1139.png,COVID/images/COVID-1139.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1139.png,COVID/masks/COVID-1139.png,True,False,299,299,L,149.92,1564768efeb7cd1ed603bf172f79ab24
COVID-114.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-114.png,COVID/images/COVID-114.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-114.png,COVID/masks/COVID-114.png,True,False,299,299,L,147.06,4a44994f4bc7af9c2f3968d51eb45616
COVID-1140.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1140.png,COVID/images/COVID-1140.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1140.png,COVID/masks/COVID-1140.png,True,False,299,299,L,121.53,6ddc7cbcfcc02225af0870945bcb3aad
COVID-1141.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1141.png,COVID/images/COVID-1141.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1141.png,COVID/masks/COVID-1141.png,True,False,299,299,L,167.48,675ec43a1d289b7232a806946423da1d
COVID-1142.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1142.png,COVID/images/COVID-1142.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1142.png,COVID/masks/COVID-1142.png,True,False,299,299,L,158.84,06c32ed2e72ed0247cf47048b1e8ab1f
COVID-1143.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1143.png,COVID/images/COVID-1143.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1143.png,COVID/masks/COVID-1143.png,True,False,299,299,L,104.76,e257855a645edc3329662a82abecd850
COVID-1144.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1144.png,COVID/images/COVID-1144.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1144.png,COVID/masks/COVID-1144.png,True,False,299,299,L,83.38,caf56fdea69de2162588719012d3c9ee
COVID-1145.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1145.png,COVID/images/COVID-1145.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1145.png,COVID/masks/COVID-1145.png,True,False,299,299,L,107.68,d8f0ee96c5f0d2599a9cd277a466df21
COVID-1146.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1146.png,COVID/images/COVID-1146.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1146.png,COVID/masks/COVID-1146.png,True,False,299,299,L,160.91,7f36205a51eacd82f8b4b10e3a81382c
COVID-1147.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1147.png,COVID/images/COVID-1147.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1147.png,COVID/masks/COVID-1147.png,True,False,299,299,L,140.53,787d54b8634d9e4103f0fa83a2fbf611
COVID-1148.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1148.png,COVID/images/COVID-1148.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1148.png,COVID/masks/COVID-1148.png,True,False,299,299,L,160.79,967a879815d2321348091a637e0641c9
COVID-1149.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1149.png,COVID/images/COVID-1149.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1149.png,COVID/masks/COVID-1149.png,True,False,299,299,L,142.28,b9983782a95227df887c9e0a1196bca7
COVID-115.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-115.png,COVID/images/COVID-115.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-115.png,COVID/masks/COVID-115.png,True,False,299,299,L,68.82,6f19fd53121bace7f0f9422fae02220e
COVID-1150.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1150.png,COVID/images/COVID-1150.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1150.png,COVID/masks/COVID-1150.png,True,False,299,299,L,155.06,0f941e2fd49b4ec84daca6869b45bdba
COVID-1151.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1151.png,COVID/images/COVID-1151.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1151.png,COVID/masks/COVID-1151.png,True,False,299,299,L,144.79,b877971defd62475077054280f74d12f
COVID-1152.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1152.png,COVID/images/COVID-1152.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1152.png,COVID/masks/COVID-1152.png,True,False,299,299,L,169.67,df441a5e1f6f95e59255cb148ad6566b
COVID-1153.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1153.png,COVID/images/COVID-1153.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1153.png,COVID/masks/COVID-1153.png,True,False,299,299,L,129.56,1d29556d703095198f72ebac69c65f20
COVID-1154.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1154.png,COVID/images/COVID-1154.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1154.png,COVID/masks/COVID-1154.png,True,False,299,299,L,163.78,b76c968e9b3347e3235d83c1b5faa594
COVID-1155.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1155.png,COVID/images/COVID-1155.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1155.png,COVID/masks/COVID-1155.png,True,False,299,299,L,135.77,c1e697205870da62fd595bfe669f8803
COVID-1156.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1156.png,COVID/images/COVID-1156.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1156.png,COVID/masks/COVID-1156.png,True,False,299,299,L,152.77,32e8aaed19f319c7ae9513b2598b12c3
COVID-1157.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1157.png,COVID/images/COVID-1157.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1157.png,COVID/masks/COVID-1157.png,True,False,299,299,L,163.19,f376b79fdd04e9d98282f0ed40cc6a2e
COVID-1158.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1158.png,COVID/images/COVID-1158.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1158.png,COVID/masks/COVID-1158.png,True,False,299,299,L,157.72,50eae0eeb484a139fe984e5680a5f1c2
COVID-1159.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1159.png,COVID/images/COVID-1159.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1159.png,COVID/masks/COVID-1159.png,True,False,299,299,L,150.77,f30cca9e6fcef2d81a08f8dbf28b2152
COVID-116.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-116.png,COVID/images/COVID-116.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-116.png,COVID/masks/COVID-116.png,True,False,299,299,L,143.75,01d0a7a57b3f1bd2545544d4cdebe08b
COVID-1160.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1160.png,COVID/images/COVID-1160.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1160.png,COVID/masks/COVID-1160.png,True,False,299,299,L,173.55,9c8a1befc87b5442d83d70361fc98ca8
COVID-1161.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1161.png,COVID/images/COVID-1161.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1161.png,COVID/masks/COVID-1161.png,True,False,299,299,L,136.55,b304bf9e01a159738aa1c7bd6fe91e60
COVID-1162.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1162.png,COVID/images/COVID-1162.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1162.png,COVID/masks/COVID-1162.png,True,False,299,299,L,127.67,67a015c107796d96256dfbd72fa5fb31
COVID-1163.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1163.png,COVID/images/COVID-1163.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1163.png,COVID/masks/COVID-1163.png,True,False,299,299,L,188.65,7a7f6dbd2e6eb05f2ca23ceb1994dfbc
COVID-1164.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1164.png,COVID/images/COVID-1164.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1164.png,COVID/masks/COVID-1164.png,True,False,299,299,L,160.07,8d5d25ecb31ffb8ab9d76fe0000f459e
COVID-1165.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1165.png,COVID/images/COVID-1165.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1165.png,COVID/masks/COVID-1165.png,True,False,299,299,L,152.72,942eb5c3e8bd86f0ea61ed0da4308c0d
COVID-1166.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1166.png,COVID/images/COVID-1166.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1166.png,COVID/masks/COVID-1166.png,True,False,299,299,L,141.11,2bec6f706645b9f60a24614074d9fc04
COVID-1167.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1167.png,COVID/images/COVID-1167.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1167.png,COVID/masks/COVID-1167.png,True,False,299,299,L,133.87,d64d204ebc6b0cc22cd71bc293468160
COVID-1168.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1168.png,COVID/images/COVID-1168.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1168.png,COVID/masks/COVID-1168.png,True,False,299,299,L,167.62,6d42f2a7e580e776123954148a3075b0
COVID-1169.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1169.png,COVID/images/COVID-1169.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1169.png,COVID/masks/COVID-1169.png,True,False,299,299,L,163.43,e3e676a4d2a1a5ecdb83a0266c495597
COVID-117.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-117.png,COVID/images/COVID-117.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-117.png,COVID/masks/COVID-117.png,True,False,299,299,L,122.88,db8956b718366177cb15895fca3fc001
COVID-1170.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1170.png,COVID/images/COVID-1170.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1170.png,COVID/masks/COVID-1170.png,True,False,299,299,L,144.01,00f8d365be16a40eb175cf5bbf45263c
COVID-1171.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1171.png,COVID/images/COVID-1171.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1171.png,COVID/masks/COVID-1171.png,True,False,299,299,L,163.46,7324b5e1d6aa857631a342ad4f7ef9bd
COVID-1172.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1172.png,COVID/images/COVID-1172.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1172.png,COVID/masks/COVID-1172.png,True,False,299,299,L,122.38,4ddad15f91ae09e7daa69de0e3553cc2
COVID-1173.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1173.png,COVID/images/COVID-1173.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1173.png,COVID/masks/COVID-1173.png,True,False,299,299,L,107.99,f71982873ca5b5af3f7522ab237423d8
COVID-1174.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1174.png,COVID/images/COVID-1174.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1174.png,COVID/masks/COVID-1174.png,True,False,299,299,L,131.66,23a124b13e27643b91f2a58a3bf6d5e8
COVID-1175.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1175.png,COVID/images/COVID-1175.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1175.png,COVID/masks/COVID-1175.png,True,False,299,299,L,148.21,212b19d71a965fa6b902a02a44370662
COVID-1176.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1176.png,COVID/images/COVID-1176.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1176.png,COVID/masks/COVID-1176.png,True,False,299,299,L,145.89,8f2434f98010bcdbf3b6b1c995f350c3
COVID-1177.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1177.png,COVID/images/COVID-1177.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1177.png,COVID/masks/COVID-1177.png,True,False,299,299,L,138.53,21ca720beafe758279ed1e4386c88501
COVID-1178.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1178.png,COVID/images/COVID-1178.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1178.png,COVID/masks/COVID-1178.png,True,False,299,299,L,138.64,b573784cab70cdb88f06298422547aaa
COVID-1179.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1179.png,COVID/images/COVID-1179.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1179.png,COVID/masks/COVID-1179.png,True,False,299,299,L,216.56,828ca059fc62fb7a7edf76705105cc32
COVID-118.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-118.png,COVID/images/COVID-118.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-118.png,COVID/masks/COVID-118.png,True,False,299,299,L,93.48,45c26926977a3e0cc29e9db92cf01bcf
COVID-1180.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1180.png,COVID/images/COVID-1180.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1180.png,COVID/masks/COVID-1180.png,True,False,299,299,L,113.14,035d3103df2497b33a462fd38f8e1761
COVID-1181.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1181.png,COVID/images/COVID-1181.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1181.png,COVID/masks/COVID-1181.png,True,False,299,299,L,143.4,31b3d04d21f8973559fc76655fd7fb19
COVID-1182.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1182.png,COVID/images/COVID-1182.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1182.png,COVID/masks/COVID-1182.png,True,False,299,299,L,154.51,be968c1586f6900c21414efe4b7403f9
COVID-1183.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1183.png,COVID/images/COVID-1183.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1183.png,COVID/masks/COVID-1183.png,True,False,299,299,L,122.5,99060b968b4116d340dfa91ad743e3a6
COVID-1184.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1184.png,COVID/images/COVID-1184.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1184.png,COVID/masks/COVID-1184.png,True,False,299,299,L,123.42,2fcd2dfe4e1cfc126ae95b65d2203a8d
COVID-1185.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1185.png,COVID/images/COVID-1185.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1185.png,COVID/masks/COVID-1185.png,True,False,299,299,L,139.4,2b6dd6cdccc24b69ec49f507d7e2e385
COVID-1186.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1186.png,COVID/images/COVID-1186.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1186.png,COVID/masks/COVID-1186.png,True,False,299,299,L,119.5,c2231f20a550f69b1c8c97a9a5064735
COVID-1187.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1187.png,COVID/images/COVID-1187.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1187.png,COVID/masks/COVID-1187.png,True,False,299,299,L,159.46,15c291194063fb476a26930c177f4905
COVID-1188.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1188.png,COVID/images/COVID-1188.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1188.png,COVID/masks/COVID-1188.png,True,False,299,299,L,153.98,2def6f314819e4cf4a3cf621e3cb8237
COVID-1189.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1189.png,COVID/images/COVID-1189.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1189.png,COVID/masks/COVID-1189.png,True,False,299,299,L,123.27,cc0e82e4687668dd4f6eeb366110bf8d
COVID-119.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-119.png,COVID/images/COVID-119.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-119.png,COVID/masks/COVID-119.png,True,False,299,299,L,148.82,96cadc73ae0184ae2cd0623340e051a5
COVID-1190.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1190.png,COVID/images/COVID-1190.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1190.png,COVID/masks/COVID-1190.png,True,False,299,299,L,167.45,03ed8e4c23a14878b96ae938dac40ac3
COVID-1191.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1191.png,COVID/images/COVID-1191.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1191.png,COVID/masks/COVID-1191.png,True,False,299,299,L,134.24,54e5361fad0b2821a9498530487af584
COVID-1192.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1192.png,COVID/images/COVID-1192.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1192.png,COVID/masks/COVID-1192.png,True,False,299,299,L,119.42,f572b0ce92b1b95eee167adca62fc85a
COVID-1193.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1193.png,COVID/images/COVID-1193.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1193.png,COVID/masks/COVID-1193.png,True,False,299,299,L,72.59,bcb5f1af87c6cec46ccbfbc0924adda4
COVID-1194.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1194.png,COVID/images/COVID-1194.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1194.png,COVID/masks/COVID-1194.png,True,False,299,299,L,156.14,0d88376be2a9e05e072c97ff644506d4
COVID-1195.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1195.png,COVID/images/COVID-1195.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1195.png,COVID/masks/COVID-1195.png,True,False,299,299,L,141.32,4b8e887e2937b82adfdf60d1581f031c
COVID-1196.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1196.png,COVID/images/COVID-1196.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1196.png,COVID/masks/COVID-1196.png,True,False,299,299,L,164.12,ae71ba147282916df866c5b2e0ecbefd
COVID-1197.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1197.png,COVID/images/COVID-1197.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1197.png,COVID/masks/COVID-1197.png,True,False,299,299,L,168.49,a033900abfaf33261e45404b95ba518f
COVID-1198.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1198.png,COVID/images/COVID-1198.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1198.png,COVID/masks/COVID-1198.png,True,False,299,299,L,133.76,002dda9d17ac4807bf7ee1c79a80eefc
COVID-1199.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1199.png,COVID/images/COVID-1199.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1199.png,COVID/masks/COVID-1199.png,True,False,299,299,L,165.87,09a26cfdd1406b6010f77e39a41088d0
COVID-12.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-12.png,COVID/images/COVID-12.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-12.png,COVID/masks/COVID-12.png,True,False,299,299,L,150.7,7caee4a0e14e60414644eaf2f0a83aac
COVID-120.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-120.png,COVID/images/COVID-120.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-120.png,COVID/masks/COVID-120.png,True,False,299,299,L,135.26,c19e12a8c53c79e7e566a02cd89bdfaf
COVID-1200.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1200.png,COVID/images/COVID-1200.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1200.png,COVID/masks/COVID-1200.png,True,False,299,299,L,151.36,7fa37c2b110aec57a78c48840ec2cce8
COVID-1201.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1201.png,COVID/images/COVID-1201.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1201.png,COVID/masks/COVID-1201.png,True,False,299,299,L,123.56,d5eb01ce2c9bae0019e5f2c55e60441c
COVID-1202.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1202.png,COVID/images/COVID-1202.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1202.png,COVID/masks/COVID-1202.png,True,False,299,299,L,131.02,1ecb1916b0284a541750d270abf5995b
COVID-1203.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1203.png,COVID/images/COVID-1203.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1203.png,COVID/masks/COVID-1203.png,True,False,299,299,L,148.48,b4da34e10bc6d3697ae581457266503b
COVID-1204.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1204.png,COVID/images/COVID-1204.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1204.png,COVID/masks/COVID-1204.png,True,False,299,299,L,153.89,7d5470dd0a6873db02f0a9938487ff2b
COVID-1205.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1205.png,COVID/images/COVID-1205.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1205.png,COVID/masks/COVID-1205.png,True,False,299,299,L,146.42,4f14d819302cc86a29f7d458e2430192
COVID-1206.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1206.png,COVID/images/COVID-1206.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1206.png,COVID/masks/COVID-1206.png,True,False,299,299,L,130.16,4c5d543bff3f08e83ecc6bcb3df273b3
COVID-1207.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1207.png,COVID/images/COVID-1207.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1207.png,COVID/masks/COVID-1207.png,True,False,299,299,L,138.34,cee42cd3774caa60cccb13147ee53283
COVID-1208.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1208.png,COVID/images/COVID-1208.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1208.png,COVID/masks/COVID-1208.png,True,False,299,299,L,162.12,a8ed38306c2757f6eb91c97a200ad760
COVID-1209.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1209.png,COVID/images/COVID-1209.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1209.png,COVID/masks/COVID-1209.png,True,False,299,299,L,141.98,3cf82e21e729dfd4811c25cf6816d923
COVID-121.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-121.png,COVID/images/COVID-121.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-121.png,COVID/masks/COVID-121.png,True,False,299,299,L,70.54,ec0ca0d3e27148352df55df1af458af1
COVID-1210.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1210.png,COVID/images/COVID-1210.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1210.png,COVID/masks/COVID-1210.png,True,False,299,299,L,106.24,dc2a148a660f3d3060bd0429162137b3
COVID-1211.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1211.png,COVID/images/COVID-1211.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1211.png,COVID/masks/COVID-1211.png,True,False,299,299,L,155.94,e788197f32c9d60d8d00c85931604b0e
COVID-1212.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1212.png,COVID/images/COVID-1212.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1212.png,COVID/masks/COVID-1212.png,True,False,299,299,L,94.55,5ffe09b891adbd14967a8dfcbc95203e
COVID-1213.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1213.png,COVID/images/COVID-1213.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1213.png,COVID/masks/COVID-1213.png,True,False,299,299,L,118.41,2cb10e1262b6223e62c8a493e4b1fa39
COVID-1214.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1214.png,COVID/images/COVID-1214.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1214.png,COVID/masks/COVID-1214.png,True,False,299,299,L,91.5,18cef21725e4ddd77719a8553ce5e9f1
COVID-1215.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1215.png,COVID/images/COVID-1215.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1215.png,COVID/masks/COVID-1215.png,True,False,299,299,L,143.27,06ea0cec8ef464ec54f9c220b87388af
COVID-1216.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1216.png,COVID/images/COVID-1216.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1216.png,COVID/masks/COVID-1216.png,True,False,299,299,L,143.13,fdac6058d993c47700841a4ceaa77112
COVID-1217.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1217.png,COVID/images/COVID-1217.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1217.png,COVID/masks/COVID-1217.png,True,False,299,299,L,157.73,3a509ecd964f021031ece925f59cea63
COVID-1218.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1218.png,COVID/images/COVID-1218.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1218.png,COVID/masks/COVID-1218.png,True,False,299,299,L,168.43,07e5721526b92350c046a5d67500ef15
COVID-1219.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1219.png,COVID/images/COVID-1219.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1219.png,COVID/masks/COVID-1219.png,True,False,299,299,L,127.7,13921a44cf72c22eeb126fbf536d72ae
COVID-122.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-122.png,COVID/images/COVID-122.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-122.png,COVID/masks/COVID-122.png,True,False,299,299,L,148.57,12bb9c568a910929db1369856cc1e76b
COVID-1220.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1220.png,COVID/images/COVID-1220.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1220.png,COVID/masks/COVID-1220.png,True,False,299,299,L,151.27,c132fd546e90e558a9f4a54bf5865368
COVID-1221.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1221.png,COVID/images/COVID-1221.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1221.png,COVID/masks/COVID-1221.png,True,False,299,299,L,148.93,7f1ed0fcfba54ef9cefef518728ee05f
COVID-1222.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1222.png,COVID/images/COVID-1222.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1222.png,COVID/masks/COVID-1222.png,True,False,299,299,L,135.82,a6b96cd7aa3860849b59078a040657dd
COVID-1223.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1223.png,COVID/images/COVID-1223.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1223.png,COVID/masks/COVID-1223.png,True,False,299,299,L,161.78,d07937beb56441f026c022cb7a417983
COVID-1224.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1224.png,COVID/images/COVID-1224.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1224.png,COVID/masks/COVID-1224.png,True,False,299,299,L,148.54,f88b3eb74e92008fa9d5d6f11c1c8436
COVID-1225.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1225.png,COVID/images/COVID-1225.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1225.png,COVID/masks/COVID-1225.png,True,False,299,299,L,140.94,24d25db205e66c82805b94f242021d87
COVID-1226.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1226.png,COVID/images/COVID-1226.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1226.png,COVID/masks/COVID-1226.png,True,False,299,299,L,100.79,a09efc21f8400b9351eb89837474d91e
COVID-1227.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1227.png,COVID/images/COVID-1227.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1227.png,COVID/masks/COVID-1227.png,True,False,299,299,L,150.59,cf1a0c177da49f05090ae0cd5a6f4cc5
COVID-1228.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1228.png,COVID/images/COVID-1228.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1228.png,COVID/masks/COVID-1228.png,True,False,299,299,L,158.36,a33f27466db74abda8b354f599a6f233
COVID-1229.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1229.png,COVID/images/COVID-1229.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1229.png,COVID/masks/COVID-1229.png,True,False,299,299,L,97.73,6cce3d8f63103bbaf1491b041e443905
COVID-123.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-123.png,COVID/images/COVID-123.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-123.png,COVID/masks/COVID-123.png,True,False,299,299,L,136.77,549a0fcfedf09ccdc5be5a50f3611ba8
COVID-1230.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1230.png,COVID/images/COVID-1230.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1230.png,COVID/masks/COVID-1230.png,True,False,299,299,L,146.46,e9d6fd0be2ed053be5d58bfe7232fd82
COVID-1231.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1231.png,COVID/images/COVID-1231.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1231.png,COVID/masks/COVID-1231.png,True,False,299,299,L,144.3,a5026809301a364475cb0ea602552c60
COVID-1232.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1232.png,COVID/images/COVID-1232.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1232.png,COVID/masks/COVID-1232.png,True,False,299,299,L,59.21,ce730df061c0f493434c9a43085f7cda
COVID-1233.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1233.png,COVID/images/COVID-1233.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1233.png,COVID/masks/COVID-1233.png,True,False,299,299,L,137.78,478c38c375855cff99e7e13a20b2624c
COVID-1234.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1234.png,COVID/images/COVID-1234.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1234.png,COVID/masks/COVID-1234.png,True,False,299,299,L,126.65,d56fb4f2904f9ea199961467507ee19b
COVID-1235.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1235.png,COVID/images/COVID-1235.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1235.png,COVID/masks/COVID-1235.png,True,False,299,299,L,164.67,1d890c47a43c7de3e2dc735ab2c4dc37
COVID-1236.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1236.png,COVID/images/COVID-1236.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1236.png,COVID/masks/COVID-1236.png,True,False,299,299,L,133.14,de7b828d77ef8925d8157f2e8695906f
COVID-1237.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1237.png,COVID/images/COVID-1237.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1237.png,COVID/masks/COVID-1237.png,True,False,299,299,L,150.0,fb805aebd50fde2d14ab0b03876b209b
COVID-1238.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1238.png,COVID/images/COVID-1238.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1238.png,COVID/masks/COVID-1238.png,True,False,299,299,L,153.22,d27602a6b99c085da83c1bea2bdeab01
COVID-1239.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1239.png,COVID/images/COVID-1239.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1239.png,COVID/masks/COVID-1239.png,True,False,299,299,L,170.08,79a6afbb28c0c6cce7f65d8226af56ad
COVID-124.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-124.png,COVID/images/COVID-124.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-124.png,COVID/masks/COVID-124.png,True,False,299,299,L,189.39,ee23d421abcb09f3d401546cd056c475
COVID-1240.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1240.png,COVID/images/COVID-1240.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1240.png,COVID/masks/COVID-1240.png,True,False,299,299,L,145.22,fb98ce115029efacec1ca7426eb21ff0
COVID-1241.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1241.png,COVID/images/COVID-1241.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1241.png,COVID/masks/COVID-1241.png,True,False,299,299,L,117.68,af132fe545c8588b605bc3090ff0bf78
COVID-1242.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1242.png,COVID/images/COVID-1242.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1242.png,COVID/masks/COVID-1242.png,True,False,299,299,L,114.48,46c66efefae8cbc1bf5dbd3b1c8e625b
COVID-1243.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1243.png,COVID/images/COVID-1243.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1243.png,COVID/masks/COVID-1243.png,True,False,299,299,L,145.18,f2eee66af45b3ef1714048faf45b51bc
COVID-1244.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1244.png,COVID/images/COVID-1244.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1244.png,COVID/masks/COVID-1244.png,True,False,299,299,L,156.88,1986b7ec4318999a76a5a55249acf432
COVID-1245.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1245.png,COVID/images/COVID-1245.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1245.png,COVID/masks/COVID-1245.png,True,False,299,299,L,136.28,0e139524a0253c5ac554df56b6bc9a53
COVID-1246.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1246.png,COVID/images/COVID-1246.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1246.png,COVID/masks/COVID-1246.png,True,False,299,299,L,130.49,db1cbb5e6381e7a8b9ad0226aa698e78
COVID-1247.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1247.png,COVID/images/COVID-1247.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1247.png,COVID/masks/COVID-1247.png,True,False,299,299,L,139.94,b0d51c5e68c7967a19ab10058056c266
COVID-1248.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1248.png,COVID/images/COVID-1248.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1248.png,COVID/masks/COVID-1248.png,True,False,299,299,L,147.9,b192201d7e2b66c37aba660b344a29e5
COVID-1249.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1249.png,COVID/images/COVID-1249.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1249.png,COVID/masks/COVID-1249.png,True,False,299,299,L,126.71,f69182973f5b1e2c5b654fec8a942f29
COVID-125.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-125.png,COVID/images/COVID-125.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-125.png,COVID/masks/COVID-125.png,True,False,299,299,L,127.57,b7fefa242770eb5094a9eb1a97e980d2
COVID-1250.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1250.png,COVID/images/COVID-1250.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1250.png,COVID/masks/COVID-1250.png,True,False,299,299,L,151.27,02767332eb598ec9bb70f0b8822d5f6b
COVID-1251.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1251.png,COVID/images/COVID-1251.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1251.png,COVID/masks/COVID-1251.png,True,False,299,299,L,153.86,a17ef39e60018601936974ec3482ae75
COVID-1252.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1252.png,COVID/images/COVID-1252.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1252.png,COVID/masks/COVID-1252.png,True,False,299,299,L,158.72,b07087349f93957274eebaf72966a7cb
COVID-1253.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1253.png,COVID/images/COVID-1253.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1253.png,COVID/masks/COVID-1253.png,True,False,299,299,L,146.73,55b6c7b4e0d076e6b9131846b1ef54ed
COVID-1254.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1254.png,COVID/images/COVID-1254.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1254.png,COVID/masks/COVID-1254.png,True,False,299,299,L,127.29,d6220f5862e5a942132461904e649f33
COVID-1255.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1255.png,COVID/images/COVID-1255.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1255.png,COVID/masks/COVID-1255.png,True,False,299,299,L,137.58,84442f7b9331f33e452f6b2e367cb165
COVID-1256.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1256.png,COVID/images/COVID-1256.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1256.png,COVID/masks/COVID-1256.png,True,False,299,299,L,127.8,610c675c71cbad1155da2d5ad90de68a
COVID-1257.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1257.png,COVID/images/COVID-1257.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1257.png,COVID/masks/COVID-1257.png,True,False,299,299,L,97.51,f3f49f043089644761b58d0e537eff4d
COVID-1258.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1258.png,COVID/images/COVID-1258.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1258.png,COVID/masks/COVID-1258.png,True,False,299,299,L,149.26,bbf17745a20ee4304c3a3222fdb51bb0
COVID-1259.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1259.png,COVID/images/COVID-1259.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1259.png,COVID/masks/COVID-1259.png,True,False,299,299,L,138.17,fb3cbc3dd1491b565e79dde7f3a5885b
COVID-126.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-126.png,COVID/images/COVID-126.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-126.png,COVID/masks/COVID-126.png,True,False,299,299,L,153.26,5ab3e886c4efe2a6e9d72d7ff1c10dd6
COVID-1260.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1260.png,COVID/images/COVID-1260.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1260.png,COVID/masks/COVID-1260.png,True,False,299,299,L,140.31,b7438a0d889940b465ca694fa6170243
COVID-1261.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1261.png,COVID/images/COVID-1261.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1261.png,COVID/masks/COVID-1261.png,True,False,299,299,L,137.27,2de0f37d9bce11b5c64d8dfa01a4b63f
COVID-1262.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1262.png,COVID/images/COVID-1262.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1262.png,COVID/masks/COVID-1262.png,True,False,299,299,L,149.99,b139a2624e4346d8bfaeaaf619a02e0a
COVID-1263.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1263.png,COVID/images/COVID-1263.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1263.png,COVID/masks/COVID-1263.png,True,False,299,299,L,164.05,9ebc4dc2fe072a79b5944ca40fbe24d4
COVID-1264.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1264.png,COVID/images/COVID-1264.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1264.png,COVID/masks/COVID-1264.png,True,False,299,299,L,164.06,ce1fabbe6c2d2877b8419940ed20ffb6
COVID-1265.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1265.png,COVID/images/COVID-1265.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1265.png,COVID/masks/COVID-1265.png,True,False,299,299,L,121.6,2a6d05760115b8df46f85d5f6618cccf
COVID-1266.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1266.png,COVID/images/COVID-1266.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1266.png,COVID/masks/COVID-1266.png,True,False,299,299,L,141.13,139e43f62de43ac1ce2fafd37543b7cc
COVID-1267.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1267.png,COVID/images/COVID-1267.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1267.png,COVID/masks/COVID-1267.png,True,False,299,299,L,139.86,acd8db3217164b704dfce0d3520e6087
COVID-1268.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1268.png,COVID/images/COVID-1268.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1268.png,COVID/masks/COVID-1268.png,True,False,299,299,L,114.96,689938b2b0828501992d7428db92329c
COVID-1269.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1269.png,COVID/images/COVID-1269.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1269.png,COVID/masks/COVID-1269.png,True,False,299,299,L,150.47,38fddd1d0a2aef2643afd790dfe7ac2f
COVID-127.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-127.png,COVID/images/COVID-127.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-127.png,COVID/masks/COVID-127.png,True,False,299,299,L,190.02,db54ac5dcc1c7b6c749db25d29fb86a7
COVID-1270.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1270.png,COVID/images/COVID-1270.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1270.png,COVID/masks/COVID-1270.png,True,False,299,299,L,136.12,387dd21c0b6959d09bb7a1715a700d70
COVID-1271.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1271.png,COVID/images/COVID-1271.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1271.png,COVID/masks/COVID-1271.png,True,False,299,299,L,146.53,7f09ff6744a3a651bc9407bc8b8b7d02
COVID-1272.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1272.png,COVID/images/COVID-1272.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1272.png,COVID/masks/COVID-1272.png,True,False,299,299,L,158.26,b7db35337f887b022e023dfde8f6bdf0
COVID-1273.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1273.png,COVID/images/COVID-1273.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1273.png,COVID/masks/COVID-1273.png,True,False,299,299,L,167.13,89e762e338c730e9d34b441a87ee00c9
COVID-1274.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1274.png,COVID/images/COVID-1274.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1274.png,COVID/masks/COVID-1274.png,True,False,299,299,L,149.04,32ee2981ba42ee53c9b38a6455aa33eb
COVID-1275.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1275.png,COVID/images/COVID-1275.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1275.png,COVID/masks/COVID-1275.png,True,False,299,299,L,128.15,df53671fa1febe90cb257a7a8a4f8980
COVID-1276.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1276.png,COVID/images/COVID-1276.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1276.png,COVID/masks/COVID-1276.png,True,False,299,299,L,159.09,1fc1eb0dbba953df731c91b1a0b2f9ca
COVID-1277.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1277.png,COVID/images/COVID-1277.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1277.png,COVID/masks/COVID-1277.png,True,False,299,299,L,160.17,51bc9eecd1db77503ad6665168bac20f
COVID-1278.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1278.png,COVID/images/COVID-1278.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1278.png,COVID/masks/COVID-1278.png,True,False,299,299,L,150.41,54201a76def047959b77f96bb55d6bc1
COVID-1279.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1279.png,COVID/images/COVID-1279.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1279.png,COVID/masks/COVID-1279.png,True,False,299,299,L,137.83,9687ecdcc0e47b274659c857794fb2ee
COVID-128.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-128.png,COVID/images/COVID-128.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-128.png,COVID/masks/COVID-128.png,True,False,299,299,L,164.43,e7ab9beb8862b4d95047aec009d01e27
COVID-1280.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1280.png,COVID/images/COVID-1280.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1280.png,COVID/masks/COVID-1280.png,True,False,299,299,L,109.72,4fe18179c3dd2a48aff5793f1adf9a92
COVID-1281.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1281.png,COVID/images/COVID-1281.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1281.png,COVID/masks/COVID-1281.png,True,False,299,299,L,182.63,b758a7a88b41f95d2b01b1a664e46d98
COVID-1282.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1282.png,COVID/images/COVID-1282.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1282.png,COVID/masks/COVID-1282.png,True,False,299,299,L,179.32,faaeeac870287ce8d43a78a75822a947
COVID-1283.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1283.png,COVID/images/COVID-1283.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1283.png,COVID/masks/COVID-1283.png,True,False,299,299,L,106.42,0879cdfc7ac949783201bbff2bf8d75c
COVID-1284.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1284.png,COVID/images/COVID-1284.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1284.png,COVID/masks/COVID-1284.png,True,False,299,299,L,156.73,2566009e1b37586c1cb7d6c5119ecd56
COVID-1285.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1285.png,COVID/images/COVID-1285.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1285.png,COVID/masks/COVID-1285.png,True,False,299,299,L,141.72,7e4a66a5f465c5e1749e5786b2c2962f
COVID-1286.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1286.png,COVID/images/COVID-1286.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1286.png,COVID/masks/COVID-1286.png,True,False,299,299,L,123.77,dfcb23ace31c521462c68e11a6569a93
COVID-1287.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1287.png,COVID/images/COVID-1287.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1287.png,COVID/masks/COVID-1287.png,True,False,299,299,L,181.5,6ec5429c97db735c5fd030e81f24aaf4
COVID-1288.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1288.png,COVID/images/COVID-1288.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1288.png,COVID/masks/COVID-1288.png,True,False,299,299,L,150.94,9875fcde732acf2f61e3d687a01945c2
COVID-1289.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1289.png,COVID/images/COVID-1289.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1289.png,COVID/masks/COVID-1289.png,True,False,299,299,L,109.28,b87e9a5e2cb5c0c45cd1236c6a04cb2b
COVID-129.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-129.png,COVID/images/COVID-129.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-129.png,COVID/masks/COVID-129.png,True,False,299,299,L,135.28,d2114dc98e7ca482b41aaa12fbdb1832
COVID-1290.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1290.png,COVID/images/COVID-1290.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1290.png,COVID/masks/COVID-1290.png,True,False,299,299,L,164.71,3bc48a54c698108bb5e69ca8074ddaba
COVID-1291.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1291.png,COVID/images/COVID-1291.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1291.png,COVID/masks/COVID-1291.png,True,False,299,299,L,109.8,087d549f0dab465558706d0a9c91a169
COVID-1292.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1292.png,COVID/images/COVID-1292.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1292.png,COVID/masks/COVID-1292.png,True,False,299,299,L,158.52,b9431650c5ac614b6c8c323cab60443a
COVID-1293.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1293.png,COVID/images/COVID-1293.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1293.png,COVID/masks/COVID-1293.png,True,False,299,299,L,140.71,171c74db6ab3c0f4acda40c485fa30d6
COVID-1294.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1294.png,COVID/images/COVID-1294.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1294.png,COVID/masks/COVID-1294.png,True,False,299,299,L,152.77,073b8a0b27d2c6ed191ccada0828c0ab
COVID-1295.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1295.png,COVID/images/COVID-1295.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1295.png,COVID/masks/COVID-1295.png,True,False,299,299,L,156.59,8220d431ba32411c3e2d106737c84554
COVID-1296.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1296.png,COVID/images/COVID-1296.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1296.png,COVID/masks/COVID-1296.png,True,False,299,299,L,126.23,aa23b159e796309079642e8fc7d578d0
COVID-1297.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1297.png,COVID/images/COVID-1297.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1297.png,COVID/masks/COVID-1297.png,True,False,299,299,L,155.46,def2af888c7e129f47bcb06412bd9357
COVID-1298.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1298.png,COVID/images/COVID-1298.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1298.png,COVID/masks/COVID-1298.png,True,False,299,299,L,149.51,2a4aa520cff25c9ab72a26f162c3e24a
COVID-1299.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1299.png,COVID/images/COVID-1299.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1299.png,COVID/masks/COVID-1299.png,True,False,299,299,L,131.11,feeece6fe5f6406e47f31f4aaa3313e3
COVID-13.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-13.png,COVID/images/COVID-13.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-13.png,COVID/masks/COVID-13.png,True,False,299,299,L,120.63,982261eb78353b6e38bc7d048902891d
COVID-130.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-130.png,COVID/images/COVID-130.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-130.png,COVID/masks/COVID-130.png,True,False,299,299,L,173.73,e3bfc96f1afee24dfeda164c523cb43b
COVID-1300.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1300.png,COVID/images/COVID-1300.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1300.png,COVID/masks/COVID-1300.png,True,False,299,299,L,146.37,9e14e00ebf3399ac09422e592fe5aaea
COVID-1301.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1301.png,COVID/images/COVID-1301.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1301.png,COVID/masks/COVID-1301.png,True,False,299,299,L,153.97,0440b965a6ee668ff11680e6126eb57a
COVID-1302.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1302.png,COVID/images/COVID-1302.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1302.png,COVID/masks/COVID-1302.png,True,False,299,299,L,167.32,a2a463766535ce6c6248ac39c46b21dd
COVID-1303.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1303.png,COVID/images/COVID-1303.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1303.png,COVID/masks/COVID-1303.png,True,False,299,299,L,158.79,3850def98e2538627e17466d8e6c8693
COVID-1304.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1304.png,COVID/images/COVID-1304.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1304.png,COVID/masks/COVID-1304.png,True,False,299,299,L,144.06,c4256cc19b1e4481900970e30f40dee5
COVID-1305.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1305.png,COVID/images/COVID-1305.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1305.png,COVID/masks/COVID-1305.png,True,False,299,299,L,149.09,e7f6750b3db7549829552fc9a08c553e
COVID-1306.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1306.png,COVID/images/COVID-1306.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1306.png,COVID/masks/COVID-1306.png,True,False,299,299,L,99.44,88f72930cb6dc7c35b6a0250624a403e
COVID-1307.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1307.png,COVID/images/COVID-1307.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1307.png,COVID/masks/COVID-1307.png,True,False,299,299,L,167.83,0515f8fcd1a555ba0d5a9809749dda5b
COVID-1308.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1308.png,COVID/images/COVID-1308.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1308.png,COVID/masks/COVID-1308.png,True,False,299,299,L,167.39,1ad9e06ab76b7758395e357926d011ae
COVID-1309.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1309.png,COVID/images/COVID-1309.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1309.png,COVID/masks/COVID-1309.png,True,False,299,299,L,162.54,156a6c95a919c43f673b58a4a3c21ded
COVID-131.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-131.png,COVID/images/COVID-131.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-131.png,COVID/masks/COVID-131.png,True,False,299,299,L,116.53,25d007bd8c624913838c810d04bf76d2
COVID-1310.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1310.png,COVID/images/COVID-1310.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1310.png,COVID/masks/COVID-1310.png,True,False,299,299,L,143.94,e5b836ec28d9a75b0089688a35008e0b
COVID-1311.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1311.png,COVID/images/COVID-1311.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1311.png,COVID/masks/COVID-1311.png,True,False,299,299,L,165.42,2445c4a468fd9ef3f60198161b783957
COVID-1312.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1312.png,COVID/images/COVID-1312.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1312.png,COVID/masks/COVID-1312.png,True,False,299,299,L,143.35,38f2991b00893332c92836933c70c45a
COVID-1313.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1313.png,COVID/images/COVID-1313.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1313.png,COVID/masks/COVID-1313.png,True,False,299,299,L,67.75,d1ab5eafdd959d27ae9f494274476fc4
COVID-1314.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1314.png,COVID/images/COVID-1314.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1314.png,COVID/masks/COVID-1314.png,True,False,299,299,L,146.58,7d11eaac49db88ae981fc8562a6d414f
COVID-1315.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1315.png,COVID/images/COVID-1315.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1315.png,COVID/masks/COVID-1315.png,True,False,299,299,L,101.76,b45684ad2f2d9f8392b9132f76f10fe4
COVID-1316.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1316.png,COVID/images/COVID-1316.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1316.png,COVID/masks/COVID-1316.png,True,False,299,299,L,134.72,387e151f60b0d6d575d11853fa71e02f
COVID-1317.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1317.png,COVID/images/COVID-1317.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1317.png,COVID/masks/COVID-1317.png,True,False,299,299,L,145.35,24ce57499e38a635869a9835be80b7c6
COVID-1318.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1318.png,COVID/images/COVID-1318.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1318.png,COVID/masks/COVID-1318.png,True,False,299,299,L,101.97,8a0e3b24cb7ce058fb13445f9b6f509a
COVID-1319.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1319.png,COVID/images/COVID-1319.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1319.png,COVID/masks/COVID-1319.png,True,False,299,299,L,140.57,1a7fc174cff1a56affed6957d70a6e19
COVID-132.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-132.png,COVID/images/COVID-132.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-132.png,COVID/masks/COVID-132.png,True,False,299,299,L,118.44,ad43c70db69f846195a92768e6bd40a7
COVID-1320.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1320.png,COVID/images/COVID-1320.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1320.png,COVID/masks/COVID-1320.png,True,False,299,299,L,147.59,3174c8738137940c70162e3185007ba3
COVID-1321.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1321.png,COVID/images/COVID-1321.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1321.png,COVID/masks/COVID-1321.png,True,False,299,299,L,101.59,17accd76d797ac32dff5cd651484a6e7
COVID-1322.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1322.png,COVID/images/COVID-1322.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1322.png,COVID/masks/COVID-1322.png,True,False,299,299,L,54.06,fd086e73418a14652a1c45bd221a47a7
COVID-1323.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1323.png,COVID/images/COVID-1323.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1323.png,COVID/masks/COVID-1323.png,True,False,299,299,L,153.49,12f4e27d576a3fe12aa7a5e5ac991d12
COVID-1324.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1324.png,COVID/images/COVID-1324.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1324.png,COVID/masks/COVID-1324.png,True,False,299,299,L,144.37,d74ba546c6e00e64e2ba9fcd249e7714
COVID-1325.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1325.png,COVID/images/COVID-1325.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1325.png,COVID/masks/COVID-1325.png,True,False,299,299,L,120.45,dbbe342b252f0f445028d7fbeb2fe46e
COVID-1326.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1326.png,COVID/images/COVID-1326.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1326.png,COVID/masks/COVID-1326.png,True,False,299,299,L,154.33,d82396000257a736a8e33d8af6e1a246
COVID-1327.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1327.png,COVID/images/COVID-1327.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1327.png,COVID/masks/COVID-1327.png,True,False,299,299,L,108.52,b61898fea6f7be8734895ea8a23f94df
COVID-1328.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1328.png,COVID/images/COVID-1328.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1328.png,COVID/masks/COVID-1328.png,True,False,299,299,L,128.95,16a446731c28781a891632ca5c850844
COVID-1329.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1329.png,COVID/images/COVID-1329.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1329.png,COVID/masks/COVID-1329.png,True,False,299,299,L,130.77,361e73f76187345abfc333ebd020e8da
COVID-133.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-133.png,COVID/images/COVID-133.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-133.png,COVID/masks/COVID-133.png,True,False,299,299,L,126.59,4cfa4925b51fdb985e12822d6f237627
COVID-1330.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1330.png,COVID/images/COVID-1330.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1330.png,COVID/masks/COVID-1330.png,True,False,299,299,L,142.37,9bb27ba0ec14b06f0592fd0bea4ac5a6
COVID-1331.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1331.png,COVID/images/COVID-1331.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1331.png,COVID/masks/COVID-1331.png,True,False,299,299,L,126.42,27e82fbf215997fe9b11cdabeb417beb
COVID-1332.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1332.png,COVID/images/COVID-1332.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1332.png,COVID/masks/COVID-1332.png,True,False,299,299,L,152.99,b3f311c7351436f08f5703c0f9df27e1
COVID-1333.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1333.png,COVID/images/COVID-1333.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1333.png,COVID/masks/COVID-1333.png,True,False,299,299,L,161.92,92d101dc1f5d839430eb203566780f1d
COVID-1334.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1334.png,COVID/images/COVID-1334.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1334.png,COVID/masks/COVID-1334.png,True,False,299,299,L,192.34,a37516070d8ff3c97f395a5fc5ca4c9e
COVID-1335.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1335.png,COVID/images/COVID-1335.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1335.png,COVID/masks/COVID-1335.png,True,False,299,299,L,171.92,25cb2746588e224eac8495cc9e0dcc65
COVID-1336.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1336.png,COVID/images/COVID-1336.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1336.png,COVID/masks/COVID-1336.png,True,False,299,299,L,98.49,3c5e8f1b88d06f7732985971a0204862
COVID-1337.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1337.png,COVID/images/COVID-1337.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1337.png,COVID/masks/COVID-1337.png,True,False,299,299,L,163.13,8f6413447454e2159f8b3668539ba2cf
COVID-1338.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1338.png,COVID/images/COVID-1338.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1338.png,COVID/masks/COVID-1338.png,True,False,299,299,L,114.78,2cc9fcc61cc162649565683741a5be90
COVID-1339.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1339.png,COVID/images/COVID-1339.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1339.png,COVID/masks/COVID-1339.png,True,False,299,299,L,161.08,965d776c1e3ed37162d065738c94c559
COVID-134.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-134.png,COVID/images/COVID-134.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-134.png,COVID/masks/COVID-134.png,True,False,299,299,L,141.81,53a420f54f56561904864346834ae191
COVID-1340.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1340.png,COVID/images/COVID-1340.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1340.png,COVID/masks/COVID-1340.png,True,False,299,299,L,125.66,e91233cf8f8f33075a635d8140066a84
COVID-1341.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1341.png,COVID/images/COVID-1341.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1341.png,COVID/masks/COVID-1341.png,True,False,299,299,L,114.5,bdc027a341d6418539885208fb5b6fef
COVID-1342.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1342.png,COVID/images/COVID-1342.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1342.png,COVID/masks/COVID-1342.png,True,False,299,299,L,126.42,badde6337e1890afe67f0dedc9748250
COVID-1343.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1343.png,COVID/images/COVID-1343.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1343.png,COVID/masks/COVID-1343.png,True,False,299,299,L,152.97,fa9202d745c33c79415253d7b7065732
COVID-1344.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1344.png,COVID/images/COVID-1344.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1344.png,COVID/masks/COVID-1344.png,True,False,299,299,L,140.63,701581040437176e26704ecbc7f5cddc
COVID-1345.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1345.png,COVID/images/COVID-1345.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1345.png,COVID/masks/COVID-1345.png,True,False,299,299,L,149.52,2800e203a87131ae9182a6f3716feafc
COVID-1346.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1346.png,COVID/images/COVID-1346.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1346.png,COVID/masks/COVID-1346.png,True,False,299,299,L,143.27,eec17629d03be199720f0717d4e9b31e
COVID-1347.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1347.png,COVID/images/COVID-1347.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1347.png,COVID/masks/COVID-1347.png,True,False,299,299,L,107.6,ed50dc1cba0ff5ea97d61ea8603c34bb
COVID-1348.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1348.png,COVID/images/COVID-1348.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1348.png,COVID/masks/COVID-1348.png,True,False,299,299,L,101.96,454503161dd7106e4c16ff339af0ac76
COVID-1349.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1349.png,COVID/images/COVID-1349.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1349.png,COVID/masks/COVID-1349.png,True,False,299,299,L,139.76,0dbccf79905c05211aeb71c8a1ad49ae
COVID-135.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-135.png,COVID/images/COVID-135.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-135.png,COVID/masks/COVID-135.png,True,False,299,299,L,154.07,b65f4d64ca62ca965c08813d5cf0cfde
COVID-1350.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1350.png,COVID/images/COVID-1350.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1350.png,COVID/masks/COVID-1350.png,True,False,299,299,L,146.33,157eff0f0c31956b3638ff2f2abfe727
COVID-1351.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1351.png,COVID/images/COVID-1351.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1351.png,COVID/masks/COVID-1351.png,True,False,299,299,L,141.42,c39baa266c17db3766934d5cd3f10f98
COVID-1352.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1352.png,COVID/images/COVID-1352.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1352.png,COVID/masks/COVID-1352.png,True,False,299,299,L,132.59,d901158e301dcb5ed2dbfdf29cfde5c5
COVID-1353.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1353.png,COVID/images/COVID-1353.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1353.png,COVID/masks/COVID-1353.png,True,False,299,299,L,145.04,acf71f15330265200800516cdd4ced93
COVID-1354.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1354.png,COVID/images/COVID-1354.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1354.png,COVID/masks/COVID-1354.png,True,False,299,299,L,137.23,7641c74d1a5dc2308641ebb53bb07b7a
COVID-1355.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1355.png,COVID/images/COVID-1355.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1355.png,COVID/masks/COVID-1355.png,True,False,299,299,L,116.33,dbc646428051db76a524de4fd788d629
COVID-1356.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1356.png,COVID/images/COVID-1356.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1356.png,COVID/masks/COVID-1356.png,True,False,299,299,L,119.66,6e24959ff0eb6de2a71b49619ef6445b
COVID-1357.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1357.png,COVID/images/COVID-1357.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1357.png,COVID/masks/COVID-1357.png,True,False,299,299,L,164.32,e2ac5af8a344f6b6f7a58996a803a052
COVID-1358.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1358.png,COVID/images/COVID-1358.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1358.png,COVID/masks/COVID-1358.png,True,False,299,299,L,155.5,f7bb9f5b779d0019b079db022d5ce70a
COVID-1359.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1359.png,COVID/images/COVID-1359.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1359.png,COVID/masks/COVID-1359.png,True,False,299,299,L,152.75,25332c2a711fdadda510ab14ef05f1e0
COVID-136.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-136.png,COVID/images/COVID-136.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-136.png,COVID/masks/COVID-136.png,True,False,299,299,L,126.03,3fb12910fd8456f030da31c64c4448f9
COVID-1360.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1360.png,COVID/images/COVID-1360.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1360.png,COVID/masks/COVID-1360.png,True,False,299,299,L,102.26,b019c2b026a9bbbc2d9990ccb63ed5c6
COVID-1361.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1361.png,COVID/images/COVID-1361.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1361.png,COVID/masks/COVID-1361.png,True,False,299,299,L,163.02,4af3f789e05d94c301dbd3a66759f8ac
COVID-1362.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1362.png,COVID/images/COVID-1362.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1362.png,COVID/masks/COVID-1362.png,True,False,299,299,L,158.36,7fb8735ccd6c19f6d97f5d70a4abc7ac
COVID-1363.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1363.png,COVID/images/COVID-1363.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1363.png,COVID/masks/COVID-1363.png,True,False,299,299,L,105.71,0e98ccc552cdb44053f472f185f0e876
COVID-1364.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1364.png,COVID/images/COVID-1364.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1364.png,COVID/masks/COVID-1364.png,True,False,299,299,L,153.93,c96372f5a8e7eee16f95dd827b3c2d8b
COVID-1365.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1365.png,COVID/images/COVID-1365.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1365.png,COVID/masks/COVID-1365.png,True,False,299,299,L,99.64,39437954cd7b1122616996c387248194
COVID-1366.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1366.png,COVID/images/COVID-1366.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1366.png,COVID/masks/COVID-1366.png,True,False,299,299,L,149.65,013cfd6e0bfe0b764026d81d47ebc482
COVID-1367.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1367.png,COVID/images/COVID-1367.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1367.png,COVID/masks/COVID-1367.png,True,False,299,299,L,107.78,197813fc0a56530e7d5d383cab591b9a
COVID-1368.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1368.png,COVID/images/COVID-1368.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1368.png,COVID/masks/COVID-1368.png,True,False,299,299,L,168.12,90b3b2a2950bf22378cf43995cb8770c
COVID-1369.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1369.png,COVID/images/COVID-1369.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1369.png,COVID/masks/COVID-1369.png,True,False,299,299,L,149.54,f3f27834413e8d7d6d77e383717657d9
COVID-137.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-137.png,COVID/images/COVID-137.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-137.png,COVID/masks/COVID-137.png,True,False,299,299,L,168.32,5cc65acef3c47f8c2999b9fa98c9369b
COVID-1370.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1370.png,COVID/images/COVID-1370.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1370.png,COVID/masks/COVID-1370.png,True,False,299,299,L,136.82,721fa939269b3faf84225e8202873c49
COVID-1371.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1371.png,COVID/images/COVID-1371.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1371.png,COVID/masks/COVID-1371.png,True,False,299,299,L,147.03,eb96d0ffd7cab4b6dbceb90f396726a5
COVID-1372.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1372.png,COVID/images/COVID-1372.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1372.png,COVID/masks/COVID-1372.png,True,False,299,299,L,188.24,fe956ced92557f17c9cef555d2540d62
COVID-1373.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1373.png,COVID/images/COVID-1373.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1373.png,COVID/masks/COVID-1373.png,True,False,299,299,L,146.02,df2f227992c1f7bc66f4b8216a28e240
COVID-1374.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1374.png,COVID/images/COVID-1374.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1374.png,COVID/masks/COVID-1374.png,True,False,299,299,L,134.35,133e1f20fe2bf432deed3e7428f86b48
COVID-1375.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1375.png,COVID/images/COVID-1375.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1375.png,COVID/masks/COVID-1375.png,True,False,299,299,L,142.9,d2c70e8a895eb19ee8c82768e57cef5c
COVID-1376.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1376.png,COVID/images/COVID-1376.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1376.png,COVID/masks/COVID-1376.png,True,False,299,299,L,158.43,83dca9402fa24b52682cbe571f45d626
COVID-1377.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1377.png,COVID/images/COVID-1377.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1377.png,COVID/masks/COVID-1377.png,True,False,299,299,L,148.04,9f0687867ac3e1c072abae3be9ab8078
COVID-1378.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1378.png,COVID/images/COVID-1378.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1378.png,COVID/masks/COVID-1378.png,True,False,299,299,L,152.44,f48403e4eafb77354e65e137766ef370
COVID-1379.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1379.png,COVID/images/COVID-1379.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1379.png,COVID/masks/COVID-1379.png,True,False,299,299,L,134.35,b18a8360d9c909369bb1e13aececbae8
COVID-138.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-138.png,COVID/images/COVID-138.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-138.png,COVID/masks/COVID-138.png,True,False,299,299,L,121.79,fc9dc3a10d292cbc7b5ded974c177c88
COVID-1380.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1380.png,COVID/images/COVID-1380.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1380.png,COVID/masks/COVID-1380.png,True,False,299,299,L,117.09,e12b35d85f04913e4bbdb2e9d9c3ae46
COVID-1381.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1381.png,COVID/images/COVID-1381.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1381.png,COVID/masks/COVID-1381.png,True,False,299,299,L,165.7,0765e166d5287d327221f0da667a939f
COVID-1382.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1382.png,COVID/images/COVID-1382.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1382.png,COVID/masks/COVID-1382.png,True,False,299,299,L,160.42,29a98592b27d908aa6a1789b8bae2516
COVID-1383.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1383.png,COVID/images/COVID-1383.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1383.png,COVID/masks/COVID-1383.png,True,False,299,299,L,160.68,e3898f052888d6e9b3c1996eb423c953
COVID-1384.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1384.png,COVID/images/COVID-1384.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1384.png,COVID/masks/COVID-1384.png,True,False,299,299,L,95.09,b12fa2c231546a493d662abd279bdc53
COVID-1385.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1385.png,COVID/images/COVID-1385.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1385.png,COVID/masks/COVID-1385.png,True,False,299,299,L,145.53,f69853381b67a2214f85dfdbce0f5b84
COVID-1386.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1386.png,COVID/images/COVID-1386.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1386.png,COVID/masks/COVID-1386.png,True,False,299,299,L,117.68,303c648eb5e169f1a48b32db75d36b22
COVID-1387.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1387.png,COVID/images/COVID-1387.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1387.png,COVID/masks/COVID-1387.png,True,False,299,299,L,144.48,d2284e41813603a9fdc3673a73d68037
COVID-1388.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1388.png,COVID/images/COVID-1388.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1388.png,COVID/masks/COVID-1388.png,True,False,299,299,L,162.46,b6d3782c19129e121e1bbd4decce006c
COVID-1389.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1389.png,COVID/images/COVID-1389.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1389.png,COVID/masks/COVID-1389.png,True,False,299,299,L,164.46,99313e3e919653f5bfd9226ebaeade09
COVID-139.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-139.png,COVID/images/COVID-139.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-139.png,COVID/masks/COVID-139.png,True,False,299,299,L,168.32,5cc65acef3c47f8c2999b9fa98c9369b
COVID-1390.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1390.png,COVID/images/COVID-1390.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1390.png,COVID/masks/COVID-1390.png,True,False,299,299,L,157.35,aa03b082c2a1364bbd30edf43557103f
COVID-1391.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1391.png,COVID/images/COVID-1391.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1391.png,COVID/masks/COVID-1391.png,True,False,299,299,L,144.08,f37244d992054a6f6758c7e37fc2b30e
COVID-1392.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1392.png,COVID/images/COVID-1392.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1392.png,COVID/masks/COVID-1392.png,True,False,299,299,L,150.55,1f8244008a1067df9ab1cef485af2d58
COVID-1393.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1393.png,COVID/images/COVID-1393.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1393.png,COVID/masks/COVID-1393.png,True,False,299,299,L,145.75,b39c8c7fa19ad95cee62b03d4deadc20
COVID-1394.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1394.png,COVID/images/COVID-1394.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1394.png,COVID/masks/COVID-1394.png,True,False,299,299,L,132.31,644adfa177cc6df8b7f869e9afe9a254
COVID-1395.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1395.png,COVID/images/COVID-1395.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1395.png,COVID/masks/COVID-1395.png,True,False,299,299,L,149.68,e7b660d7f35d3c9c9f164d47ea68e58c
COVID-1396.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1396.png,COVID/images/COVID-1396.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1396.png,COVID/masks/COVID-1396.png,True,False,299,299,L,159.44,26f45bd88f12be220b6c89dd50fd5d2f
COVID-1397.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1397.png,COVID/images/COVID-1397.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1397.png,COVID/masks/COVID-1397.png,True,False,299,299,L,165.07,ef5cb4543a5a7aa95e39d59f755c6734
COVID-1398.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1398.png,COVID/images/COVID-1398.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1398.png,COVID/masks/COVID-1398.png,True,False,299,299,L,159.41,6879bb6a2d7b0bc130b897fb4ad3de19
COVID-1399.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1399.png,COVID/images/COVID-1399.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1399.png,COVID/masks/COVID-1399.png,True,False,299,299,L,159.35,5df5bb16e592d9c9b2f511c15d44bf10
COVID-14.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-14.png,COVID/images/COVID-14.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-14.png,COVID/masks/COVID-14.png,True,False,299,299,L,124.47,7c12039720e57338dc54ba691c2efae2
COVID-140.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-140.png,COVID/images/COVID-140.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-140.png,COVID/masks/COVID-140.png,True,False,299,299,L,146.4,7b0d682211f32513201bed8cede77d11
COVID-1400.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1400.png,COVID/images/COVID-1400.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1400.png,COVID/masks/COVID-1400.png,True,False,299,299,L,178.26,1c7fe5b6e646057115575e24e02dbf2f
COVID-1401.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1401.png,COVID/images/COVID-1401.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1401.png,COVID/masks/COVID-1401.png,True,False,299,299,L,165.01,616f35953b7e8e5b89de7751d6c00559
COVID-1402.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1402.png,COVID/images/COVID-1402.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1402.png,COVID/masks/COVID-1402.png,True,False,299,299,L,165.07,7d9d991fa33a64ee7fd3e81a21894cfd
COVID-1403.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1403.png,COVID/images/COVID-1403.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1403.png,COVID/masks/COVID-1403.png,True,False,299,299,L,126.87,8cdd96266011be852d9ccd013fd52682
COVID-1404.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1404.png,COVID/images/COVID-1404.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1404.png,COVID/masks/COVID-1404.png,True,False,299,299,L,181.95,2d9da8e7d90a4c26e68b694b3156181f
COVID-1405.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1405.png,COVID/images/COVID-1405.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1405.png,COVID/masks/COVID-1405.png,True,False,299,299,L,144.14,0360307239984739d0712c2f5dc815cb
COVID-1406.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1406.png,COVID/images/COVID-1406.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1406.png,COVID/masks/COVID-1406.png,True,False,299,299,L,172.39,37e87c591d99d307c3f22fabba64a63f
COVID-1407.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1407.png,COVID/images/COVID-1407.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1407.png,COVID/masks/COVID-1407.png,True,False,299,299,L,149.99,fdaa59fa3c8c56b2a1bb598654cc75dd
COVID-1408.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1408.png,COVID/images/COVID-1408.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1408.png,COVID/masks/COVID-1408.png,True,False,299,299,L,106.49,c6cae37b4c6132ecd7a7d0b7ce28c7f6
COVID-1409.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1409.png,COVID/images/COVID-1409.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1409.png,COVID/masks/COVID-1409.png,True,False,299,299,L,169.69,c597707457ddd4df31e62703ad8efcc2
COVID-141.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-141.png,COVID/images/COVID-141.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-141.png,COVID/masks/COVID-141.png,True,False,299,299,L,159.0,53f474b92ade89065f0ac87df9a7c3ea
COVID-1410.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1410.png,COVID/images/COVID-1410.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1410.png,COVID/masks/COVID-1410.png,True,False,299,299,L,153.0,99d69a0d9521d1df8cccf23c953554ec
COVID-1411.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1411.png,COVID/images/COVID-1411.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1411.png,COVID/masks/COVID-1411.png,True,False,299,299,L,155.55,c8cc986cf00a09bcd3aa91346fa4b531
COVID-1412.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1412.png,COVID/images/COVID-1412.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1412.png,COVID/masks/COVID-1412.png,True,False,299,299,L,181.32,bf6ebc597fc530b98bec8fadff1dfd21
COVID-1413.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1413.png,COVID/images/COVID-1413.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1413.png,COVID/masks/COVID-1413.png,True,False,299,299,L,167.47,43f7d0d1385c2c3f40de84d0591331ef
COVID-1414.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1414.png,COVID/images/COVID-1414.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1414.png,COVID/masks/COVID-1414.png,True,False,299,299,L,143.14,339c6d9ff42a89387a5c33839f6b5643
COVID-1415.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1415.png,COVID/images/COVID-1415.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1415.png,COVID/masks/COVID-1415.png,True,False,299,299,L,145.82,a43087e1800f6e57c0f66f9974f12297
COVID-1416.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1416.png,COVID/images/COVID-1416.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1416.png,COVID/masks/COVID-1416.png,True,False,299,299,L,164.72,497e1992b44f02620aa7498dad185f60
COVID-1417.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1417.png,COVID/images/COVID-1417.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1417.png,COVID/masks/COVID-1417.png,True,False,299,299,L,162.74,5410363b5670297f86b3c191fbc2fc38
COVID-1418.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1418.png,COVID/images/COVID-1418.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1418.png,COVID/masks/COVID-1418.png,True,False,299,299,L,145.01,6d446cb178758cbd92a587358d74bae8
COVID-1419.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1419.png,COVID/images/COVID-1419.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1419.png,COVID/masks/COVID-1419.png,True,False,299,299,L,83.37,7258448750c3f1d998001f9dcf99d435
COVID-142.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-142.png,COVID/images/COVID-142.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-142.png,COVID/masks/COVID-142.png,True,False,299,299,L,132.39,2eb358cb4b77a481d3a8fbf19666a230
COVID-1420.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1420.png,COVID/images/COVID-1420.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1420.png,COVID/masks/COVID-1420.png,True,False,299,299,L,99.73,e7d8ad46bf2e177daa1463db29825d75
COVID-1421.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1421.png,COVID/images/COVID-1421.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1421.png,COVID/masks/COVID-1421.png,True,False,299,299,L,167.67,31e6b5d02d0825bbc8b8ed5574e894c8
COVID-1422.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1422.png,COVID/images/COVID-1422.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1422.png,COVID/masks/COVID-1422.png,True,False,299,299,L,117.44,80fb6cbcb831bab45d18ccd5e4dd0a6b
COVID-1423.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1423.png,COVID/images/COVID-1423.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1423.png,COVID/masks/COVID-1423.png,True,False,299,299,L,143.98,016b690ee11a412063a7b0c9ebe85eb9
COVID-1424.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1424.png,COVID/images/COVID-1424.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1424.png,COVID/masks/COVID-1424.png,True,False,299,299,L,158.58,63c51318a88f9a471c9467d3d0fb1516
COVID-1425.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1425.png,COVID/images/COVID-1425.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1425.png,COVID/masks/COVID-1425.png,True,False,299,299,L,121.7,51b7f1465d880d18494efbdbc01c09c5
COVID-1426.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1426.png,COVID/images/COVID-1426.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1426.png,COVID/masks/COVID-1426.png,True,False,299,299,L,103.96,1417d15025e7f5d356da371eb76b90db
COVID-1427.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1427.png,COVID/images/COVID-1427.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1427.png,COVID/masks/COVID-1427.png,True,False,299,299,L,169.19,d6045a8d9dd9fa2e7d248b041ffdee2e
COVID-1428.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1428.png,COVID/images/COVID-1428.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1428.png,COVID/masks/COVID-1428.png,True,False,299,299,L,110.47,1a71168021b3d073e7f00ade5d77e121
COVID-1429.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1429.png,COVID/images/COVID-1429.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1429.png,COVID/masks/COVID-1429.png,True,False,299,299,L,149.68,8b1d8307a2d3628db25eaec2798e47d5
COVID-143.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-143.png,COVID/images/COVID-143.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-143.png,COVID/masks/COVID-143.png,True,False,299,299,L,168.04,e8f0c9f566a20ae24a2d982e8a571478
COVID-1430.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1430.png,COVID/images/COVID-1430.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1430.png,COVID/masks/COVID-1430.png,True,False,299,299,L,169.39,f19dca25b9bc6e918fa08c3f793ac888
COVID-1431.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1431.png,COVID/images/COVID-1431.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1431.png,COVID/masks/COVID-1431.png,True,False,299,299,L,151.49,4be7a6fe5214a4ae5aa5cf4bd5bb60c1
COVID-1432.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1432.png,COVID/images/COVID-1432.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1432.png,COVID/masks/COVID-1432.png,True,False,299,299,L,139.54,ac77b48c900d453a68952f35a5595897
COVID-1433.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1433.png,COVID/images/COVID-1433.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1433.png,COVID/masks/COVID-1433.png,True,False,299,299,L,171.01,67fd1b00a2645ec015053330ea69b6ef
COVID-1434.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1434.png,COVID/images/COVID-1434.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1434.png,COVID/masks/COVID-1434.png,True,False,299,299,L,112.28,43fd1496191d9ff8eef4aba04733b48d
COVID-1435.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1435.png,COVID/images/COVID-1435.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1435.png,COVID/masks/COVID-1435.png,True,False,299,299,L,141.68,e9588e768e91cad5e4e6a34ec520f8fa
COVID-1436.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1436.png,COVID/images/COVID-1436.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1436.png,COVID/masks/COVID-1436.png,True,False,299,299,L,72.78,66fe1eb9938459b918e28f7559cf4d83
COVID-1437.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1437.png,COVID/images/COVID-1437.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1437.png,COVID/masks/COVID-1437.png,True,False,299,299,L,156.88,16b8da1e61bce294aaa7dd8f98f9f8eb
COVID-1438.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1438.png,COVID/images/COVID-1438.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1438.png,COVID/masks/COVID-1438.png,True,False,299,299,L,130.97,06c2b651b241438f88930daa6b9d9a35
COVID-1439.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1439.png,COVID/images/COVID-1439.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1439.png,COVID/masks/COVID-1439.png,True,False,299,299,L,182.2,91fcad414579025cc7cb1c5686c5ba58
COVID-144.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-144.png,COVID/images/COVID-144.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-144.png,COVID/masks/COVID-144.png,True,False,299,299,L,140.75,fd01420b756b638b56fdf710951c9c66
COVID-1440.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1440.png,COVID/images/COVID-1440.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1440.png,COVID/masks/COVID-1440.png,True,False,299,299,L,151.82,141368bfed63ce7eeb7a417bf215dffd
COVID-1441.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1441.png,COVID/images/COVID-1441.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1441.png,COVID/masks/COVID-1441.png,True,False,299,299,L,156.95,7c3337cb2e92f55e619f8d0e80761429
COVID-1442.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1442.png,COVID/images/COVID-1442.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1442.png,COVID/masks/COVID-1442.png,True,False,299,299,L,153.16,2fef51e9d0a866673abc56c92a06cc44
COVID-1443.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1443.png,COVID/images/COVID-1443.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1443.png,COVID/masks/COVID-1443.png,True,False,299,299,L,141.17,98803b992692c9c82320198c313d0874
COVID-1444.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1444.png,COVID/images/COVID-1444.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1444.png,COVID/masks/COVID-1444.png,True,False,299,299,L,164.1,858b93dd11b1d5f37a52973866c26248
COVID-1445.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1445.png,COVID/images/COVID-1445.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1445.png,COVID/masks/COVID-1445.png,True,False,299,299,L,151.97,ce82d96646e4e946df7b888aac04a202
COVID-1446.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1446.png,COVID/images/COVID-1446.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1446.png,COVID/masks/COVID-1446.png,True,False,299,299,L,136.12,bf5ef1428ff2563c523b383c2e326a86
COVID-1447.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1447.png,COVID/images/COVID-1447.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1447.png,COVID/masks/COVID-1447.png,True,False,299,299,L,106.45,733ff7dbc13e41ba78ad5d5f1891ed7c
COVID-1448.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1448.png,COVID/images/COVID-1448.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1448.png,COVID/masks/COVID-1448.png,True,False,299,299,L,151.97,0572e77d60214e68e3c102b659441210
COVID-1449.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1449.png,COVID/images/COVID-1449.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1449.png,COVID/masks/COVID-1449.png,True,False,299,299,L,167.33,a8e32a15ee2ba1a4f358be3a14ae113c
COVID-145.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-145.png,COVID/images/COVID-145.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-145.png,COVID/masks/COVID-145.png,True,False,299,299,L,154.03,13766f635aed65dea7fb80c5aa45da48
COVID-1450.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1450.png,COVID/images/COVID-1450.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1450.png,COVID/masks/COVID-1450.png,True,False,299,299,L,161.05,643b61965f723dec3796ba852dbc538c
COVID-1451.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1451.png,COVID/images/COVID-1451.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1451.png,COVID/masks/COVID-1451.png,True,False,299,299,L,174.32,315c15a4339da159de0d347c8ee6d138
COVID-1452.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1452.png,COVID/images/COVID-1452.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1452.png,COVID/masks/COVID-1452.png,True,False,299,299,L,148.64,d331b759a2670d4521e14e01c86975a3
COVID-1453.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1453.png,COVID/images/COVID-1453.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1453.png,COVID/masks/COVID-1453.png,True,False,299,299,L,125.7,784a711f1898160837a5fc38281676fe
COVID-1454.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1454.png,COVID/images/COVID-1454.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1454.png,COVID/masks/COVID-1454.png,True,False,299,299,L,171.65,80118e99dbf72e1744a14b41374a0d5f
COVID-1455.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1455.png,COVID/images/COVID-1455.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1455.png,COVID/masks/COVID-1455.png,True,False,299,299,L,150.25,446c4039288a6bb6b12217be7a795ad4
COVID-1456.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1456.png,COVID/images/COVID-1456.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1456.png,COVID/masks/COVID-1456.png,True,False,299,299,L,157.72,825763c6d109f627f1edc19472033346
COVID-1457.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1457.png,COVID/images/COVID-1457.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1457.png,COVID/masks/COVID-1457.png,True,False,299,299,L,163.24,2c75359d50b04fb6fc2045fb7303449e
COVID-1458.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1458.png,COVID/images/COVID-1458.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1458.png,COVID/masks/COVID-1458.png,True,False,299,299,L,126.96,6af1dd86a70e3068a320b66caedb2b19
COVID-1459.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1459.png,COVID/images/COVID-1459.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1459.png,COVID/masks/COVID-1459.png,True,False,299,299,L,153.64,71a80ea675553f9dc98f20d88c2696c0
COVID-146.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-146.png,COVID/images/COVID-146.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-146.png,COVID/masks/COVID-146.png,True,False,299,299,L,137.27,2b87dda723daaf80bcdbb3c7e081eab7
COVID-1460.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1460.png,COVID/images/COVID-1460.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1460.png,COVID/masks/COVID-1460.png,True,False,299,299,L,145.54,e7aea9e3249faa40c8768a34f272c58f
COVID-1461.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1461.png,COVID/images/COVID-1461.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1461.png,COVID/masks/COVID-1461.png,True,False,299,299,L,130.76,ad5a2a9a6962c4be264b9328e4495c5c
COVID-1462.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1462.png,COVID/images/COVID-1462.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1462.png,COVID/masks/COVID-1462.png,True,False,299,299,L,103.75,ffc5b0e1f37d6163fb9499efd2b117f9
COVID-1463.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1463.png,COVID/images/COVID-1463.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1463.png,COVID/masks/COVID-1463.png,True,False,299,299,L,147.88,54f5e4cd8a54bc43d61829e23843a5dc
COVID-1464.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1464.png,COVID/images/COVID-1464.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1464.png,COVID/masks/COVID-1464.png,True,False,299,299,L,151.32,f6a8237df865eea1957255d808bfae5b
COVID-1465.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1465.png,COVID/images/COVID-1465.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1465.png,COVID/masks/COVID-1465.png,True,False,299,299,L,131.1,c89f423ab472b300a5fb34390a223e21
COVID-1466.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1466.png,COVID/images/COVID-1466.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1466.png,COVID/masks/COVID-1466.png,True,False,299,299,L,105.3,a6f6707e3f5fea84e29afeb2ff73f057
COVID-1467.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1467.png,COVID/images/COVID-1467.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1467.png,COVID/masks/COVID-1467.png,True,False,299,299,L,138.36,bebf45ae92fc87e7f5e600443e6c8195
COVID-1468.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1468.png,COVID/images/COVID-1468.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1468.png,COVID/masks/COVID-1468.png,True,False,299,299,L,189.96,62b10f71df6cf53895f98bab6f0f3dc9
COVID-1469.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1469.png,COVID/images/COVID-1469.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1469.png,COVID/masks/COVID-1469.png,True,False,299,299,L,118.41,01025d0e3c6fdb37bc4dc6d1b6ec87b6
COVID-147.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-147.png,COVID/images/COVID-147.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-147.png,COVID/masks/COVID-147.png,True,False,299,299,L,148.55,5326a366a38dcc93ae1e03397f27bb8e
COVID-1470.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1470.png,COVID/images/COVID-1470.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1470.png,COVID/masks/COVID-1470.png,True,False,299,299,L,165.8,67aa7a49a612fd4caed0f30b822816dc
COVID-1471.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1471.png,COVID/images/COVID-1471.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1471.png,COVID/masks/COVID-1471.png,True,False,299,299,L,148.52,a950ac4fa4199c18040419cb8a008999
COVID-1472.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1472.png,COVID/images/COVID-1472.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1472.png,COVID/masks/COVID-1472.png,True,False,299,299,L,132.77,160bf3971c3e9713ad72d2f1cea40fe5
COVID-1473.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1473.png,COVID/images/COVID-1473.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1473.png,COVID/masks/COVID-1473.png,True,False,299,299,L,154.3,a2a1a1342cf308b8b157dc2503acf8b9
COVID-1474.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1474.png,COVID/images/COVID-1474.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1474.png,COVID/masks/COVID-1474.png,True,False,299,299,L,158.69,674e8049557549f273559d941d635d19
COVID-1475.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1475.png,COVID/images/COVID-1475.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1475.png,COVID/masks/COVID-1475.png,True,False,299,299,L,145.38,0a6ff67fe264303a4d56c4746a80aa35
COVID-1476.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1476.png,COVID/images/COVID-1476.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1476.png,COVID/masks/COVID-1476.png,True,False,299,299,L,66.06,eac4ef7c3ddfc4b9e5bf838515589192
COVID-1477.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1477.png,COVID/images/COVID-1477.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1477.png,COVID/masks/COVID-1477.png,True,False,299,299,L,148.1,fff8b012a2c891d0424671aed05b17e2
COVID-1478.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1478.png,COVID/images/COVID-1478.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1478.png,COVID/masks/COVID-1478.png,True,False,299,299,L,163.02,8947d159a6832b83a50a4e3477b646bc
COVID-1479.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1479.png,COVID/images/COVID-1479.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1479.png,COVID/masks/COVID-1479.png,True,False,299,299,L,97.95,6d4f56a0b96e3375d1d15bb874aa7afa
COVID-148.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-148.png,COVID/images/COVID-148.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-148.png,COVID/masks/COVID-148.png,True,False,299,299,L,137.52,fabc30726a5269e0f9c350ac87bd74dc
COVID-1480.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1480.png,COVID/images/COVID-1480.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1480.png,COVID/masks/COVID-1480.png,True,False,299,299,L,166.12,1f78a9680529ff8c30382f03e93e07fd
COVID-1481.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1481.png,COVID/images/COVID-1481.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1481.png,COVID/masks/COVID-1481.png,True,False,299,299,L,128.18,b9f3b0166fa6789cf79e7c255eea49a2
COVID-1482.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1482.png,COVID/images/COVID-1482.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1482.png,COVID/masks/COVID-1482.png,True,False,299,299,L,136.36,756f6359c72dd8e9679d5926a3ba9882
COVID-1483.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1483.png,COVID/images/COVID-1483.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1483.png,COVID/masks/COVID-1483.png,True,False,299,299,L,190.47,44e5c3cea78875e7499fc439142928b8
COVID-1484.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1484.png,COVID/images/COVID-1484.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1484.png,COVID/masks/COVID-1484.png,True,False,299,299,L,140.9,ae970bb07f531364bed0e066d8da6321
COVID-1485.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1485.png,COVID/images/COVID-1485.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1485.png,COVID/masks/COVID-1485.png,True,False,299,299,L,125.68,d5f6b44094b9207e66f5d19148c6e2a8
COVID-1486.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1486.png,COVID/images/COVID-1486.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1486.png,COVID/masks/COVID-1486.png,True,False,299,299,L,136.8,2baedf6651a17df9c0403cccefa985ac
COVID-1487.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1487.png,COVID/images/COVID-1487.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1487.png,COVID/masks/COVID-1487.png,True,False,299,299,L,134.62,3fc58f74a433ae258b89322587be91b3
COVID-1488.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1488.png,COVID/images/COVID-1488.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1488.png,COVID/masks/COVID-1488.png,True,False,299,299,L,157.0,5312a2f88f8a6bf3a5aaea49a0b18d91
COVID-1489.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1489.png,COVID/images/COVID-1489.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1489.png,COVID/masks/COVID-1489.png,True,False,299,299,L,164.6,ab2bdb23913442e6372df206acbea34c
COVID-149.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-149.png,COVID/images/COVID-149.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-149.png,COVID/masks/COVID-149.png,True,False,299,299,L,148.53,8bed62eca3fdbf589e81f836c7b2f43c
COVID-1490.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1490.png,COVID/images/COVID-1490.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1490.png,COVID/masks/COVID-1490.png,True,False,299,299,L,148.41,1857be53602dce194c6cde5cdbc645ec
COVID-1491.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1491.png,COVID/images/COVID-1491.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1491.png,COVID/masks/COVID-1491.png,True,False,299,299,L,171.21,75f1ca0b8d472e1a6c7a2e1444436c0e
COVID-1492.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1492.png,COVID/images/COVID-1492.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1492.png,COVID/masks/COVID-1492.png,True,False,299,299,L,134.31,7c52eb02c084071131c2aa0199f57c5d
COVID-1493.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1493.png,COVID/images/COVID-1493.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1493.png,COVID/masks/COVID-1493.png,True,False,299,299,L,171.23,624e04ebbc5db40ce5003306468fcd63
COVID-1494.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1494.png,COVID/images/COVID-1494.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1494.png,COVID/masks/COVID-1494.png,True,False,299,299,L,141.02,e779a4ea68d8a92039b79f114e74f254
COVID-1495.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1495.png,COVID/images/COVID-1495.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1495.png,COVID/masks/COVID-1495.png,True,False,299,299,L,143.88,a9e4a292a291dfe58eb8679432d5eda2
COVID-1496.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1496.png,COVID/images/COVID-1496.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1496.png,COVID/masks/COVID-1496.png,True,False,299,299,L,187.14,f8d8d6fc589473e8583d9e7944b829d7
COVID-1497.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1497.png,COVID/images/COVID-1497.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1497.png,COVID/masks/COVID-1497.png,True,False,299,299,L,157.36,59e3a96c647293373b22ebdcbc11a0da
COVID-1498.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1498.png,COVID/images/COVID-1498.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1498.png,COVID/masks/COVID-1498.png,True,False,299,299,L,171.06,4266532497938c11a3c0080341890121
COVID-1499.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1499.png,COVID/images/COVID-1499.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1499.png,COVID/masks/COVID-1499.png,True,False,299,299,L,117.51,4c9d97bfaa11423e9f29ba8a3f1802f0
COVID-15.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-15.png,COVID/images/COVID-15.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-15.png,COVID/masks/COVID-15.png,True,False,299,299,L,158.42,49cf4d6e32935cdccdabea90ff6ff444
COVID-150.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-150.png,COVID/images/COVID-150.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-150.png,COVID/masks/COVID-150.png,True,False,299,299,L,129.99,349a70af0748a3fa422e9e60fc2878ad
COVID-1500.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1500.png,COVID/images/COVID-1500.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1500.png,COVID/masks/COVID-1500.png,True,False,299,299,L,119.21,d0b1eafaa7c5489375d56d1c8962d582
COVID-1501.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1501.png,COVID/images/COVID-1501.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1501.png,COVID/masks/COVID-1501.png,True,False,299,299,L,146.96,4c8f08063f097ce0f88e99190468ecee
COVID-1502.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1502.png,COVID/images/COVID-1502.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1502.png,COVID/masks/COVID-1502.png,True,False,299,299,L,164.59,f0f1a1deb6a3cd6f130f877a35e5a871
COVID-1503.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1503.png,COVID/images/COVID-1503.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1503.png,COVID/masks/COVID-1503.png,True,False,299,299,L,145.12,7edee20094cac3168b39e5954758b5c4
COVID-1504.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1504.png,COVID/images/COVID-1504.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1504.png,COVID/masks/COVID-1504.png,True,False,299,299,L,157.31,9b5483978da573767660b80fcdc887df
COVID-1505.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1505.png,COVID/images/COVID-1505.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1505.png,COVID/masks/COVID-1505.png,True,False,299,299,L,141.99,64b4053ab3da694d27236d919792af73
COVID-1506.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1506.png,COVID/images/COVID-1506.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1506.png,COVID/masks/COVID-1506.png,True,False,299,299,L,100.24,7aa7dbfbd919c0741cd0faa2eb3b30a6
COVID-1507.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1507.png,COVID/images/COVID-1507.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1507.png,COVID/masks/COVID-1507.png,True,False,299,299,L,143.66,11ad4d19236c48ae955235628bcf7f05
COVID-1508.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1508.png,COVID/images/COVID-1508.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1508.png,COVID/masks/COVID-1508.png,True,False,299,299,L,132.02,3a4bd4384728c3b183be5ee63b67ae66
COVID-1509.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1509.png,COVID/images/COVID-1509.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1509.png,COVID/masks/COVID-1509.png,True,False,299,299,L,102.91,4e9401cdf35a3764b629f6d18b0aacc2
COVID-151.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-151.png,COVID/images/COVID-151.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-151.png,COVID/masks/COVID-151.png,True,False,299,299,L,147.68,4fbef19ad2fe8c7835752ba3c5bed46a
COVID-1510.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1510.png,COVID/images/COVID-1510.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1510.png,COVID/masks/COVID-1510.png,True,False,299,299,L,144.22,c1a9729df5ffc4b2d87bac6f56f381bf
COVID-1511.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1511.png,COVID/images/COVID-1511.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1511.png,COVID/masks/COVID-1511.png,True,False,299,299,L,147.29,46e579f48bf66bc969b36e681bff1967
COVID-1512.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1512.png,COVID/images/COVID-1512.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1512.png,COVID/masks/COVID-1512.png,True,False,299,299,L,130.91,93656ae557d99faa03ed642f4839b4de
COVID-1513.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1513.png,COVID/images/COVID-1513.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1513.png,COVID/masks/COVID-1513.png,True,False,299,299,L,116.03,d76a24b867cfaa4638ecd5c7a9465b66
COVID-1514.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1514.png,COVID/images/COVID-1514.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1514.png,COVID/masks/COVID-1514.png,True,False,299,299,L,141.47,39dfb09294a80a23fe6aef56877ef5b3
COVID-1515.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1515.png,COVID/images/COVID-1515.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1515.png,COVID/masks/COVID-1515.png,True,False,299,299,L,143.36,017e8b31c2cbf0f645f68af6c4d4e2ad
COVID-1516.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1516.png,COVID/images/COVID-1516.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1516.png,COVID/masks/COVID-1516.png,True,False,299,299,L,158.79,9efb6644ce70ea2700a1e9bd5c9ed83c
COVID-1517.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1517.png,COVID/images/COVID-1517.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1517.png,COVID/masks/COVID-1517.png,True,False,299,299,L,131.5,a76801d8fd8f6020218cc6a85196a686
COVID-1518.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1518.png,COVID/images/COVID-1518.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1518.png,COVID/masks/COVID-1518.png,True,False,299,299,L,107.22,3d8b03cacece492adba91acc830529cb
COVID-1519.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1519.png,COVID/images/COVID-1519.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1519.png,COVID/masks/COVID-1519.png,True,False,299,299,L,167.31,198413f56682c12e36659afa0ef43859
COVID-152.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-152.png,COVID/images/COVID-152.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-152.png,COVID/masks/COVID-152.png,True,False,299,299,L,186.16,daa35f520f8fc4c3f74968ff94b65587
COVID-1520.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1520.png,COVID/images/COVID-1520.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1520.png,COVID/masks/COVID-1520.png,True,False,299,299,L,159.41,8630beac5b8a0d12b431a5f8c0fd6df3
COVID-1521.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1521.png,COVID/images/COVID-1521.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1521.png,COVID/masks/COVID-1521.png,True,False,299,299,L,154.96,66b882a967556024836a4cee583679a6
COVID-1522.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1522.png,COVID/images/COVID-1522.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1522.png,COVID/masks/COVID-1522.png,True,False,299,299,L,139.54,94a96c2bd2e0209260d90b4de35049a5
COVID-1523.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1523.png,COVID/images/COVID-1523.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1523.png,COVID/masks/COVID-1523.png,True,False,299,299,L,143.42,c88510a87adcdfaba28418913bf2dae8
COVID-1524.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1524.png,COVID/images/COVID-1524.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1524.png,COVID/masks/COVID-1524.png,True,False,299,299,L,165.34,82a9281acf17ca6a6f0f8f1af15cde6b
COVID-1525.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1525.png,COVID/images/COVID-1525.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1525.png,COVID/masks/COVID-1525.png,True,False,299,299,L,152.27,b8967f8388dcd719239ab9941b3fb07d
COVID-1526.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1526.png,COVID/images/COVID-1526.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1526.png,COVID/masks/COVID-1526.png,True,False,299,299,L,111.08,01c82d394bd4dda7d4605e0b04cf04df
COVID-1527.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1527.png,COVID/images/COVID-1527.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1527.png,COVID/masks/COVID-1527.png,True,False,299,299,L,144.68,23465235fe7ef1823c90cabf1d1a99ab
COVID-1528.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1528.png,COVID/images/COVID-1528.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1528.png,COVID/masks/COVID-1528.png,True,False,299,299,L,198.52,7e7460ba82c74fd2538646ce6c57124e
COVID-1529.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1529.png,COVID/images/COVID-1529.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1529.png,COVID/masks/COVID-1529.png,True,False,299,299,L,129.3,6c4552be0327ebd25992680e2ceaa327
COVID-153.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-153.png,COVID/images/COVID-153.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-153.png,COVID/masks/COVID-153.png,True,False,299,299,L,151.05,74b1eab5519c1e6c15d2660321328f29
COVID-1530.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1530.png,COVID/images/COVID-1530.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1530.png,COVID/masks/COVID-1530.png,True,False,299,299,L,133.63,7d4dd63fe74a62154acb67518d2cacb3
COVID-1531.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1531.png,COVID/images/COVID-1531.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1531.png,COVID/masks/COVID-1531.png,True,False,299,299,L,211.36,ff00f453029b6f11ad0bed4dad1abf13
COVID-1532.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1532.png,COVID/images/COVID-1532.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1532.png,COVID/masks/COVID-1532.png,True,False,299,299,L,189.6,d63101b16793b56c4c78640e75554cf4
COVID-1533.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1533.png,COVID/images/COVID-1533.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1533.png,COVID/masks/COVID-1533.png,True,False,299,299,L,135.85,436186eb15a37346243f6a87fb18bf95
COVID-1534.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1534.png,COVID/images/COVID-1534.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1534.png,COVID/masks/COVID-1534.png,True,False,299,299,L,150.25,39e37ee3d04f416ae32bce9bcb8d1f01
COVID-1535.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1535.png,COVID/images/COVID-1535.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1535.png,COVID/masks/COVID-1535.png,True,False,299,299,L,195.53,3ab1238aff20d2c8fbf93c021f373b79
COVID-1536.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1536.png,COVID/images/COVID-1536.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1536.png,COVID/masks/COVID-1536.png,True,False,299,299,L,159.01,2591443e5329198eb4f13e53f7ab173d
COVID-1537.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1537.png,COVID/images/COVID-1537.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1537.png,COVID/masks/COVID-1537.png,True,False,299,299,L,150.31,b8d11d0f06aaf62ee81c17ecb4b01458
COVID-1538.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1538.png,COVID/images/COVID-1538.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1538.png,COVID/masks/COVID-1538.png,True,False,299,299,L,175.1,0a83d09b5d56b5c43b86c5d80ace4433
COVID-1539.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1539.png,COVID/images/COVID-1539.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1539.png,COVID/masks/COVID-1539.png,True,False,299,299,L,146.88,db41684e2b2db5eba4ddaadfa8fde62b
COVID-154.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-154.png,COVID/images/COVID-154.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-154.png,COVID/masks/COVID-154.png,True,False,299,299,L,166.23,b04b04760940d252b2a598095f35ac14
COVID-1540.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1540.png,COVID/images/COVID-1540.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1540.png,COVID/masks/COVID-1540.png,True,False,299,299,L,119.41,ebcb3dd6f0e71b969355d79ba6c73cd9
COVID-1541.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1541.png,COVID/images/COVID-1541.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1541.png,COVID/masks/COVID-1541.png,True,False,299,299,L,103.95,5d61d36349dcfca751a8445acda86d4a
COVID-1542.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1542.png,COVID/images/COVID-1542.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1542.png,COVID/masks/COVID-1542.png,True,False,299,299,L,144.89,3c085abceadcae32a289a7aabaeda9b8
COVID-1543.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1543.png,COVID/images/COVID-1543.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1543.png,COVID/masks/COVID-1543.png,True,False,299,299,L,170.89,224350355421a165ee5675c3689c93a7
COVID-1544.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1544.png,COVID/images/COVID-1544.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1544.png,COVID/masks/COVID-1544.png,True,False,299,299,L,137.06,dd0ee7cf9ea5cbd8fc6760afe2abf943
COVID-1545.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1545.png,COVID/images/COVID-1545.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1545.png,COVID/masks/COVID-1545.png,True,False,299,299,L,166.22,cb9da91341ce131381df4a285b7eaa2a
COVID-1546.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1546.png,COVID/images/COVID-1546.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1546.png,COVID/masks/COVID-1546.png,True,False,299,299,L,118.96,bcc62122ecb7f5dfc4f2bdab9fb4c593
COVID-1547.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1547.png,COVID/images/COVID-1547.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1547.png,COVID/masks/COVID-1547.png,True,False,299,299,L,125.55,dab8b3ea8dcaee2dd8105111a989681d
COVID-1548.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1548.png,COVID/images/COVID-1548.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1548.png,COVID/masks/COVID-1548.png,True,False,299,299,L,100.55,113923b09af0bed2f16c7e19a121a24b
COVID-1549.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1549.png,COVID/images/COVID-1549.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1549.png,COVID/masks/COVID-1549.png,True,False,299,299,L,134.12,7f0e565e07b437364d3ab09ba4f12d77
COVID-155.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-155.png,COVID/images/COVID-155.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-155.png,COVID/masks/COVID-155.png,True,False,299,299,L,166.12,dad2743b4d1f274312f0872175c8d2be
COVID-1550.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1550.png,COVID/images/COVID-1550.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1550.png,COVID/masks/COVID-1550.png,True,False,299,299,L,125.66,750e6150c385de82877fa010f7f3b14a
COVID-1551.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1551.png,COVID/images/COVID-1551.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1551.png,COVID/masks/COVID-1551.png,True,False,299,299,L,157.19,5330c1419ab3e5ff7aa373dc8c84ecc1
COVID-1552.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1552.png,COVID/images/COVID-1552.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1552.png,COVID/masks/COVID-1552.png,True,False,299,299,L,180.16,376538807cf4870fc5e24c23efc39660
COVID-1553.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1553.png,COVID/images/COVID-1553.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1553.png,COVID/masks/COVID-1553.png,True,False,299,299,L,165.01,5c1827b5e4334e775a7f2a03000d9b2f
COVID-1554.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1554.png,COVID/images/COVID-1554.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1554.png,COVID/masks/COVID-1554.png,True,False,299,299,L,138.51,e192d823d6d24a9cc83a9b63ba33b18c
COVID-1555.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1555.png,COVID/images/COVID-1555.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1555.png,COVID/masks/COVID-1555.png,True,False,299,299,L,164.26,082a849a257baba11555e12824d6484c
COVID-1556.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1556.png,COVID/images/COVID-1556.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1556.png,COVID/masks/COVID-1556.png,True,False,299,299,L,87.15,771d2a3b5c72814deaa967f11e7dbecb
COVID-1557.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1557.png,COVID/images/COVID-1557.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1557.png,COVID/masks/COVID-1557.png,True,False,299,299,L,99.19,3865eb560ab751a48654132083ce7d6a
COVID-1558.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1558.png,COVID/images/COVID-1558.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1558.png,COVID/masks/COVID-1558.png,True,False,299,299,L,152.84,463f7423dbd8a92c0e78da51a26718a4
COVID-1559.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1559.png,COVID/images/COVID-1559.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1559.png,COVID/masks/COVID-1559.png,True,False,299,299,L,99.16,b7a3741e9ec468bcf4243f6186b6cb37
COVID-156.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-156.png,COVID/images/COVID-156.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-156.png,COVID/masks/COVID-156.png,True,False,299,299,L,122.49,2743bdcef206ed04c5eb7b2688d61172
COVID-1560.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1560.png,COVID/images/COVID-1560.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1560.png,COVID/masks/COVID-1560.png,True,False,299,299,L,152.86,c3749bb6160cc50aaa37ed52768b3a35
COVID-1561.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1561.png,COVID/images/COVID-1561.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1561.png,COVID/masks/COVID-1561.png,True,False,299,299,L,153.64,e135a56a1cf82e1af43836955d806a4e
COVID-1562.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1562.png,COVID/images/COVID-1562.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1562.png,COVID/masks/COVID-1562.png,True,False,299,299,L,141.31,9618e960dacc99863da202b48268d548
COVID-1563.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1563.png,COVID/images/COVID-1563.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1563.png,COVID/masks/COVID-1563.png,True,False,299,299,L,159.49,a40d7d36400a93044c0e36ee7fd0794e
COVID-1564.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1564.png,COVID/images/COVID-1564.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1564.png,COVID/masks/COVID-1564.png,True,False,299,299,L,142.83,8a293a2101176c2d72cb3910562dd6ef
COVID-1565.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1565.png,COVID/images/COVID-1565.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1565.png,COVID/masks/COVID-1565.png,True,False,299,299,L,162.66,49f1dfcf0d385ac3bf3c971e21f13bd7
COVID-1566.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1566.png,COVID/images/COVID-1566.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1566.png,COVID/masks/COVID-1566.png,True,False,299,299,L,160.94,bdb9bddf431c10cdf8086e238eb1c12d
COVID-1567.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1567.png,COVID/images/COVID-1567.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1567.png,COVID/masks/COVID-1567.png,True,False,299,299,L,142.57,28a44d043c8f9946fb5579cb360016bc
COVID-1568.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1568.png,COVID/images/COVID-1568.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1568.png,COVID/masks/COVID-1568.png,True,False,299,299,L,98.73,7275bc3603d8d8c614cc840426cc59b9
COVID-1569.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1569.png,COVID/images/COVID-1569.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1569.png,COVID/masks/COVID-1569.png,True,False,299,299,L,159.29,c2f0a3a134d296523641588d345f95d4
COVID-157.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-157.png,COVID/images/COVID-157.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-157.png,COVID/masks/COVID-157.png,True,False,299,299,L,184.45,56589d69171497e79c68e222c5b02d5a
COVID-1570.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1570.png,COVID/images/COVID-1570.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1570.png,COVID/masks/COVID-1570.png,True,False,299,299,L,175.32,2dd55357053dcd8465d8d5d2214e827b
COVID-1571.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1571.png,COVID/images/COVID-1571.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1571.png,COVID/masks/COVID-1571.png,True,False,299,299,L,138.72,3cb0d5cfa642e267e1aa26e7de0d21c8
COVID-1572.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1572.png,COVID/images/COVID-1572.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1572.png,COVID/masks/COVID-1572.png,True,False,299,299,L,160.68,6b0621dc7e7583ec4279cb0b2897a4e9
COVID-1573.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1573.png,COVID/images/COVID-1573.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1573.png,COVID/masks/COVID-1573.png,True,False,299,299,L,160.35,1e75b623dbfeca04172cdbdebe9972ec
COVID-1574.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1574.png,COVID/images/COVID-1574.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1574.png,COVID/masks/COVID-1574.png,True,False,299,299,L,114.09,55cd18ed26ad4789c01ef54697f45e9c
COVID-1575.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1575.png,COVID/images/COVID-1575.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1575.png,COVID/masks/COVID-1575.png,True,False,299,299,L,128.52,fb60a9c4a138aaa7102f6ac02d940eed
COVID-1576.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1576.png,COVID/images/COVID-1576.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1576.png,COVID/masks/COVID-1576.png,True,False,299,299,L,158.87,1fdfc648079a3bb0523241a81bd43f5c
COVID-1577.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1577.png,COVID/images/COVID-1577.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1577.png,COVID/masks/COVID-1577.png,True,False,299,299,L,106.51,eb219ed0cafbee74e03872430f6779a0
COVID-1578.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1578.png,COVID/images/COVID-1578.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1578.png,COVID/masks/COVID-1578.png,True,False,299,299,L,142.29,31545a33777b21ae5088361c7b142bbb
COVID-1579.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1579.png,COVID/images/COVID-1579.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1579.png,COVID/masks/COVID-1579.png,True,False,299,299,L,159.73,6a72ccd181d377e6a1bca1d07db92f09
COVID-158.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-158.png,COVID/images/COVID-158.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-158.png,COVID/masks/COVID-158.png,True,False,299,299,L,160.48,ed2ac741134fd31097fe47cc2af85f13
COVID-1580.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1580.png,COVID/images/COVID-1580.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1580.png,COVID/masks/COVID-1580.png,True,False,299,299,L,152.46,4b926090de9ad7a79e3dc31b3d229883
COVID-1581.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1581.png,COVID/images/COVID-1581.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1581.png,COVID/masks/COVID-1581.png,True,False,299,299,L,156.95,83579421bc2b4cefc51e46e65910ff3c
COVID-1582.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1582.png,COVID/images/COVID-1582.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1582.png,COVID/masks/COVID-1582.png,True,False,299,299,L,157.39,da7e0136ec2ba366475e0391b7d71fc8
COVID-1583.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1583.png,COVID/images/COVID-1583.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1583.png,COVID/masks/COVID-1583.png,True,False,299,299,L,135.08,dc76cb9e40b3d431d1f8b03a5afb3879
COVID-1584.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1584.png,COVID/images/COVID-1584.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1584.png,COVID/masks/COVID-1584.png,True,False,299,299,L,98.68,d759fbda4f3581f217a63194f3ba7b93
COVID-1585.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1585.png,COVID/images/COVID-1585.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1585.png,COVID/masks/COVID-1585.png,True,False,299,299,L,174.28,fb6c891283f869cf362b085a6bbe4caa
COVID-1586.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1586.png,COVID/images/COVID-1586.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1586.png,COVID/masks/COVID-1586.png,True,False,299,299,L,117.15,0756b51406a2acb0f8ddde333aecbd1f
COVID-1587.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1587.png,COVID/images/COVID-1587.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1587.png,COVID/masks/COVID-1587.png,True,False,299,299,L,149.52,0be1ccc226b6141fe9e7ef429899dcfa
COVID-1588.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1588.png,COVID/images/COVID-1588.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1588.png,COVID/masks/COVID-1588.png,True,False,299,299,L,127.4,439957463830f0e2f108829b36f35a82
COVID-1589.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1589.png,COVID/images/COVID-1589.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1589.png,COVID/masks/COVID-1589.png,True,False,299,299,L,125.86,e93e403772cb73401e2e8be587cbc146
COVID-159.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-159.png,COVID/images/COVID-159.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-159.png,COVID/masks/COVID-159.png,True,False,299,299,L,173.46,00b09884b96575eab6629bb413f05bef
COVID-1590.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1590.png,COVID/images/COVID-1590.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1590.png,COVID/masks/COVID-1590.png,True,False,299,299,L,163.77,2a64da0969dc08b102286ed5c7410dd0
COVID-1591.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1591.png,COVID/images/COVID-1591.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1591.png,COVID/masks/COVID-1591.png,True,False,299,299,L,158.79,0773aa4716b37c0a07f10be68744f32a
COVID-1592.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1592.png,COVID/images/COVID-1592.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1592.png,COVID/masks/COVID-1592.png,True,False,299,299,L,152.22,ab7fb8ddfdcf91b5146ee26d5466a05e
COVID-1593.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1593.png,COVID/images/COVID-1593.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1593.png,COVID/masks/COVID-1593.png,True,False,299,299,L,170.83,5668f8a63ff75c4c4e5ad47f4b195a9d
COVID-1594.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1594.png,COVID/images/COVID-1594.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1594.png,COVID/masks/COVID-1594.png,True,False,299,299,L,149.91,c60aa2782d4e678b5531d12624bc5024
COVID-1595.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1595.png,COVID/images/COVID-1595.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1595.png,COVID/masks/COVID-1595.png,True,False,299,299,L,98.37,d7a9374f6650e547141f92318124b4f9
COVID-1596.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1596.png,COVID/images/COVID-1596.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1596.png,COVID/masks/COVID-1596.png,True,False,299,299,L,150.99,4f3ce11e62c63debb767971b1ce9f807
COVID-1597.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1597.png,COVID/images/COVID-1597.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1597.png,COVID/masks/COVID-1597.png,True,False,299,299,L,174.42,d97d6c540cb7c10942a10e6cd6d84ec6
COVID-1598.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1598.png,COVID/images/COVID-1598.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1598.png,COVID/masks/COVID-1598.png,True,False,299,299,L,99.34,8b13c344c79b8783e3ba367cfc0fd4c9
COVID-1599.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1599.png,COVID/images/COVID-1599.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1599.png,COVID/masks/COVID-1599.png,True,False,299,299,L,103.7,1a7a8fd5047cc3329b5713acae7e392e
COVID-16.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-16.png,COVID/images/COVID-16.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-16.png,COVID/masks/COVID-16.png,True,False,299,299,L,129.68,6e0534b84048705b7c72ba762fe31ddd
COVID-160.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-160.png,COVID/images/COVID-160.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-160.png,COVID/masks/COVID-160.png,True,False,299,299,L,149.85,9784813ed917978581430bc645763553
COVID-1600.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1600.png,COVID/images/COVID-1600.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1600.png,COVID/masks/COVID-1600.png,True,False,299,299,L,135.95,63f88a52d2ffbc5ee95cfcd6eeb04b7e
COVID-1601.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1601.png,COVID/images/COVID-1601.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1601.png,COVID/masks/COVID-1601.png,True,False,299,299,L,115.28,3b16f4c6981f02c2a2d024ce957ddb8b
COVID-1602.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1602.png,COVID/images/COVID-1602.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1602.png,COVID/masks/COVID-1602.png,True,False,299,299,L,149.74,41b652fcd9be31bca68eef9e9637cf3b
COVID-1603.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1603.png,COVID/images/COVID-1603.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1603.png,COVID/masks/COVID-1603.png,True,False,299,299,L,178.55,1194e653ec521568d968b95035af7a08
COVID-1604.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1604.png,COVID/images/COVID-1604.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1604.png,COVID/masks/COVID-1604.png,True,False,299,299,L,90.52,1c6a9e2e98886def4985d6e8fdeee305
COVID-1605.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1605.png,COVID/images/COVID-1605.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1605.png,COVID/masks/COVID-1605.png,True,False,299,299,L,101.48,5f1f389f28d1a4bdd7a6dafbda510395
COVID-1606.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1606.png,COVID/images/COVID-1606.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1606.png,COVID/masks/COVID-1606.png,True,False,299,299,L,184.56,c6d8109af83861c93a816dbddf221bb5
COVID-1607.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1607.png,COVID/images/COVID-1607.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1607.png,COVID/masks/COVID-1607.png,True,False,299,299,L,145.58,913959f5065747c98eeb625e84aae02c
COVID-1608.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1608.png,COVID/images/COVID-1608.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1608.png,COVID/masks/COVID-1608.png,True,False,299,299,L,139.19,8ec32d277c5ac91ce7ef1b06b21fe9ef
COVID-1609.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1609.png,COVID/images/COVID-1609.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1609.png,COVID/masks/COVID-1609.png,True,False,299,299,L,148.81,c17e7894d5c8aa2180eb0fe668f15f35
COVID-161.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-161.png,COVID/images/COVID-161.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-161.png,COVID/masks/COVID-161.png,True,False,299,299,L,137.72,b4a16bf40f94e6552f2c549ee42af62c
COVID-1610.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1610.png,COVID/images/COVID-1610.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1610.png,COVID/masks/COVID-1610.png,True,False,299,299,L,137.14,e47a81dae3b13b681634c67242873ef5
COVID-1611.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1611.png,COVID/images/COVID-1611.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1611.png,COVID/masks/COVID-1611.png,True,False,299,299,L,113.93,27a1d7ac9c09eb0892dad3539484bbc5
COVID-1612.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1612.png,COVID/images/COVID-1612.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1612.png,COVID/masks/COVID-1612.png,True,False,299,299,L,146.57,506865ebafda045c056240631b9f11a1
COVID-1613.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1613.png,COVID/images/COVID-1613.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1613.png,COVID/masks/COVID-1613.png,True,False,299,299,L,173.98,e496839daab4036814dbf562bdc6389a
COVID-1614.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1614.png,COVID/images/COVID-1614.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1614.png,COVID/masks/COVID-1614.png,True,False,299,299,L,161.36,0875a89fe8b85453dc5fd0640d65e51f
COVID-1615.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1615.png,COVID/images/COVID-1615.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1615.png,COVID/masks/COVID-1615.png,True,False,299,299,L,173.38,52478001826d2938f0d4a4f78cd17a82
COVID-1616.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1616.png,COVID/images/COVID-1616.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1616.png,COVID/masks/COVID-1616.png,True,False,299,299,L,102.93,d72460b2127d6fbffe9546990ea6edfa
COVID-1617.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1617.png,COVID/images/COVID-1617.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1617.png,COVID/masks/COVID-1617.png,True,False,299,299,L,149.14,083d92f179eb332ffb2af063b096da8d
COVID-1618.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1618.png,COVID/images/COVID-1618.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1618.png,COVID/masks/COVID-1618.png,True,False,299,299,L,156.22,47f19112d6b4f28cb2d89598324a1a97
COVID-1619.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1619.png,COVID/images/COVID-1619.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1619.png,COVID/masks/COVID-1619.png,True,False,299,299,L,174.29,8ee7dc0e58ab6bf1c9623d3147e39c8e
COVID-162.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-162.png,COVID/images/COVID-162.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-162.png,COVID/masks/COVID-162.png,True,False,299,299,L,167.44,aedcec60a7d9227958f0f298c3bf5e45
COVID-1620.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1620.png,COVID/images/COVID-1620.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1620.png,COVID/masks/COVID-1620.png,True,False,299,299,L,96.88,a552ec3faff8dfc3027d151120a24ac6
COVID-1621.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1621.png,COVID/images/COVID-1621.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1621.png,COVID/masks/COVID-1621.png,True,False,299,299,L,157.69,b796db034ad56b2599ea2b752bdbc34f
COVID-1622.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1622.png,COVID/images/COVID-1622.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1622.png,COVID/masks/COVID-1622.png,True,False,299,299,L,135.83,2b5abc38a621c1ea129ef0e2d7b79da2
COVID-1623.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1623.png,COVID/images/COVID-1623.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1623.png,COVID/masks/COVID-1623.png,True,False,299,299,L,140.75,768244496edcc2dc98d3a570183f81ad
COVID-1624.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1624.png,COVID/images/COVID-1624.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1624.png,COVID/masks/COVID-1624.png,True,False,299,299,L,154.94,e480afa4505a5b1a4dc9bb823609d27c
COVID-1625.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1625.png,COVID/images/COVID-1625.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1625.png,COVID/masks/COVID-1625.png,True,False,299,299,L,177.89,f7697b1affcccbaba21a9b9f1c222d6f
COVID-1626.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1626.png,COVID/images/COVID-1626.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1626.png,COVID/masks/COVID-1626.png,True,False,299,299,L,140.89,e34ff29726327645f1dda73dd869c0da
COVID-1627.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1627.png,COVID/images/COVID-1627.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1627.png,COVID/masks/COVID-1627.png,True,False,299,299,L,161.55,8d5d08a1ed03ecc9deb63b5e5a516480
COVID-1628.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1628.png,COVID/images/COVID-1628.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1628.png,COVID/masks/COVID-1628.png,True,False,299,299,L,139.24,b7c85f4564fe376bd034eebc680293e5
COVID-1629.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1629.png,COVID/images/COVID-1629.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1629.png,COVID/masks/COVID-1629.png,True,False,299,299,L,153.56,5d7e3c28f28d7dba73edf5ec4541c956
COVID-163.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-163.png,COVID/images/COVID-163.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-163.png,COVID/masks/COVID-163.png,True,False,299,299,L,180.48,fd8bdaa2b74c1cd864a375f1264d195e
COVID-1630.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1630.png,COVID/images/COVID-1630.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1630.png,COVID/masks/COVID-1630.png,True,False,299,299,L,130.69,0c7f70fb4e5e704beddb57cdf1f2f607
COVID-1631.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1631.png,COVID/images/COVID-1631.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1631.png,COVID/masks/COVID-1631.png,True,False,299,299,L,140.2,74fc67b25340eed18a487bc796e4dac0
COVID-1632.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1632.png,COVID/images/COVID-1632.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1632.png,COVID/masks/COVID-1632.png,True,False,299,299,L,164.08,1dd95eff977a11eedb54c3fea10405ec
COVID-1633.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1633.png,COVID/images/COVID-1633.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1633.png,COVID/masks/COVID-1633.png,True,False,299,299,L,172.27,60516df713f85b4d63417c72a7db4818
COVID-1634.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1634.png,COVID/images/COVID-1634.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1634.png,COVID/masks/COVID-1634.png,True,False,299,299,L,151.9,5cd0e5930d5b652c26fdfb70645926ec
COVID-1635.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1635.png,COVID/images/COVID-1635.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1635.png,COVID/masks/COVID-1635.png,True,False,299,299,L,170.36,bad0f8278866aa2a9a97a9f4e9224021
COVID-1636.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1636.png,COVID/images/COVID-1636.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1636.png,COVID/masks/COVID-1636.png,True,False,299,299,L,141.63,4c21c6f90ae5ed8b8c20587089ea5ad0
COVID-1637.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1637.png,COVID/images/COVID-1637.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1637.png,COVID/masks/COVID-1637.png,True,False,299,299,L,179.17,e82eff763e775dba91d88b51d57db10e
COVID-1638.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1638.png,COVID/images/COVID-1638.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1638.png,COVID/masks/COVID-1638.png,True,False,299,299,L,95.29,28321b885a672d6854ffbd9e9befc154
COVID-1639.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1639.png,COVID/images/COVID-1639.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1639.png,COVID/masks/COVID-1639.png,True,False,299,299,L,155.63,753cb0a2e4fb1aa118c8bd73872d2df0
COVID-164.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-164.png,COVID/images/COVID-164.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-164.png,COVID/masks/COVID-164.png,True,False,299,299,L,167.16,927dd77ef9d3d2906595ab0b678e866f
COVID-1640.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1640.png,COVID/images/COVID-1640.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1640.png,COVID/masks/COVID-1640.png,True,False,299,299,L,147.49,8630c68e8ca1c8091a7fdb7354cae96f
COVID-1641.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1641.png,COVID/images/COVID-1641.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1641.png,COVID/masks/COVID-1641.png,True,False,299,299,L,153.8,8a5ae05d17e0f94ac211dca80ce4242f
COVID-1642.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1642.png,COVID/images/COVID-1642.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1642.png,COVID/masks/COVID-1642.png,True,False,299,299,L,148.7,7db5db359d54af4fe238257b10e645fd
COVID-1643.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1643.png,COVID/images/COVID-1643.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1643.png,COVID/masks/COVID-1643.png,True,False,299,299,L,153.02,f538bc9c12e2581971d8d6f3559abad1
COVID-1644.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1644.png,COVID/images/COVID-1644.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1644.png,COVID/masks/COVID-1644.png,True,False,299,299,L,153.41,a6965b7c6935fb717f700db73b8f609e
COVID-1645.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1645.png,COVID/images/COVID-1645.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1645.png,COVID/masks/COVID-1645.png,True,False,299,299,L,164.92,f7d2dc3c3c5d31d7a488ba8d3e4eeb0e
COVID-1646.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1646.png,COVID/images/COVID-1646.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1646.png,COVID/masks/COVID-1646.png,True,False,299,299,L,99.53,5566c31782978de9e71957003a7dd226
COVID-1647.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1647.png,COVID/images/COVID-1647.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1647.png,COVID/masks/COVID-1647.png,True,False,299,299,L,167.76,1caeb0f6e3554cf9030cb23ff40cf8a3
COVID-1648.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1648.png,COVID/images/COVID-1648.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1648.png,COVID/masks/COVID-1648.png,True,False,299,299,L,117.58,91359e76aee644d5c6f5efc66e404164
COVID-1649.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1649.png,COVID/images/COVID-1649.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1649.png,COVID/masks/COVID-1649.png,True,False,299,299,L,143.88,938d8a594411cb2e22a36ab8e871b219
COVID-165.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-165.png,COVID/images/COVID-165.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-165.png,COVID/masks/COVID-165.png,True,False,299,299,L,94.89,3a47555c358a3c4e309a1475776e848b
COVID-1650.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1650.png,COVID/images/COVID-1650.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1650.png,COVID/masks/COVID-1650.png,True,False,299,299,L,139.77,58d3467ac2113899ee379c821658d55f
COVID-1651.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1651.png,COVID/images/COVID-1651.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1651.png,COVID/masks/COVID-1651.png,True,False,299,299,L,153.6,59bb237a49a0e9a97b16569d59df89bc
COVID-1652.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1652.png,COVID/images/COVID-1652.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1652.png,COVID/masks/COVID-1652.png,True,False,299,299,L,147.58,34f95933f281bba2ba7060e65290e22d
COVID-1653.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1653.png,COVID/images/COVID-1653.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1653.png,COVID/masks/COVID-1653.png,True,False,299,299,L,145.89,1fcbda6855d97745c1344a1f7c1f7343
COVID-1654.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1654.png,COVID/images/COVID-1654.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1654.png,COVID/masks/COVID-1654.png,True,False,299,299,L,158.4,85d6f55c0cacc73c6ce16983670f2bab
COVID-1655.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1655.png,COVID/images/COVID-1655.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1655.png,COVID/masks/COVID-1655.png,True,False,299,299,L,163.76,9d1bd2d448c1cb3a3f8121280b8a9cb1
COVID-1656.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1656.png,COVID/images/COVID-1656.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1656.png,COVID/masks/COVID-1656.png,True,False,299,299,L,152.55,a9c5ef5302f9c59ed1564b51c7190b63
COVID-1657.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1657.png,COVID/images/COVID-1657.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1657.png,COVID/masks/COVID-1657.png,True,False,299,299,L,173.09,3d6e9e62bccc69bd3f7ab8aafd414947
COVID-1658.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1658.png,COVID/images/COVID-1658.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1658.png,COVID/masks/COVID-1658.png,True,False,299,299,L,151.73,6396a4b5c2f46817351f8b83579d1331
COVID-1659.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1659.png,COVID/images/COVID-1659.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1659.png,COVID/masks/COVID-1659.png,True,False,299,299,L,137.64,6686c79b694fcbffe5a4e5bd40306c57
COVID-166.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-166.png,COVID/images/COVID-166.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-166.png,COVID/masks/COVID-166.png,True,False,299,299,L,136.16,1b1ac772e2ecdd0c70182355f652cd4d
COVID-1660.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1660.png,COVID/images/COVID-1660.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1660.png,COVID/masks/COVID-1660.png,True,False,299,299,L,156.8,685fc336dbf34c6bd6d5ddb8dbfa9158
COVID-1661.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1661.png,COVID/images/COVID-1661.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1661.png,COVID/masks/COVID-1661.png,True,False,299,299,L,114.02,592d18bb969b073399a3535a5c9e4504
COVID-1662.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1662.png,COVID/images/COVID-1662.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1662.png,COVID/masks/COVID-1662.png,True,False,299,299,L,115.7,789cf8ff60b97a1d0abc3f1fdcd0a0c2
COVID-1663.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1663.png,COVID/images/COVID-1663.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1663.png,COVID/masks/COVID-1663.png,True,False,299,299,L,158.57,65815c7fc7be45904cf192797b43c774
COVID-1664.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1664.png,COVID/images/COVID-1664.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1664.png,COVID/masks/COVID-1664.png,True,False,299,299,L,89.06,c2b7ceb78c9b47f99a89f326692e8f74
COVID-1665.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1665.png,COVID/images/COVID-1665.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1665.png,COVID/masks/COVID-1665.png,True,False,299,299,L,178.08,807498adae9ac77c5eba65a81316326f
COVID-1666.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1666.png,COVID/images/COVID-1666.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1666.png,COVID/masks/COVID-1666.png,True,False,299,299,L,99.19,653cccdca5bdc456d3a474ce53fea250
COVID-1667.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1667.png,COVID/images/COVID-1667.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1667.png,COVID/masks/COVID-1667.png,True,False,299,299,L,172.6,11285e2533998a712c4ea2c914bc8fd0
COVID-1668.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1668.png,COVID/images/COVID-1668.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1668.png,COVID/masks/COVID-1668.png,True,False,299,299,L,155.47,cb2ae9cd12d82dad0ae26591e7048872
COVID-1669.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1669.png,COVID/images/COVID-1669.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1669.png,COVID/masks/COVID-1669.png,True,False,299,299,L,179.72,f9296358bfd9df701796c3ee9ea13ed0
COVID-167.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-167.png,COVID/images/COVID-167.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-167.png,COVID/masks/COVID-167.png,True,False,299,299,L,164.13,56f2520cab4d15125b6228670ee34314
COVID-1670.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1670.png,COVID/images/COVID-1670.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1670.png,COVID/masks/COVID-1670.png,True,False,299,299,L,153.46,7d3188efe93a3a721e65ae7f497d8149
COVID-1671.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1671.png,COVID/images/COVID-1671.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1671.png,COVID/masks/COVID-1671.png,True,False,299,299,L,166.34,46ecd4384aecd526f1cd4fa467dba814
COVID-1672.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1672.png,COVID/images/COVID-1672.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1672.png,COVID/masks/COVID-1672.png,True,False,299,299,L,137.33,c7129092ea0b1860e3e3dc2886931d32
COVID-1673.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1673.png,COVID/images/COVID-1673.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1673.png,COVID/masks/COVID-1673.png,True,False,299,299,L,130.09,f302a31c3903c7d285f3d15defbccc28
COVID-1674.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1674.png,COVID/images/COVID-1674.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1674.png,COVID/masks/COVID-1674.png,True,False,299,299,L,127.54,b87e52a93684f8428f50d637c4341a5d
COVID-1675.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1675.png,COVID/images/COVID-1675.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1675.png,COVID/masks/COVID-1675.png,True,False,299,299,L,172.08,4af3c54c87d220540964c0dcdadf9c6d
COVID-1676.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1676.png,COVID/images/COVID-1676.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1676.png,COVID/masks/COVID-1676.png,True,False,299,299,L,99.81,7efc9017f4d239edc7a322bfb2b3b664
COVID-1677.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1677.png,COVID/images/COVID-1677.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1677.png,COVID/masks/COVID-1677.png,True,False,299,299,L,192.07,745a16a746bbdcacf29f8c7971f7ec19
COVID-1678.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1678.png,COVID/images/COVID-1678.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1678.png,COVID/masks/COVID-1678.png,True,False,299,299,L,165.0,e659b3409f8d297e84fc31a320fc194c
COVID-1679.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1679.png,COVID/images/COVID-1679.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1679.png,COVID/masks/COVID-1679.png,True,False,299,299,L,138.59,3cc2e59b0216d29728616699b5299b5d
COVID-168.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-168.png,COVID/images/COVID-168.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-168.png,COVID/masks/COVID-168.png,True,False,299,299,L,128.16,a046aa9ba6359f948ffcbbc5091077f7
COVID-1680.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1680.png,COVID/images/COVID-1680.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1680.png,COVID/masks/COVID-1680.png,True,False,299,299,L,154.48,d6f9ffe621a9a247d039ad7d55f041b6
COVID-1681.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1681.png,COVID/images/COVID-1681.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1681.png,COVID/masks/COVID-1681.png,True,False,299,299,L,120.45,1e4b9644424b7d615a78550d89b0a190
COVID-1682.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1682.png,COVID/images/COVID-1682.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1682.png,COVID/masks/COVID-1682.png,True,False,299,299,L,151.87,4bfd10d98f240e6fe7bb5062b26f5007
COVID-1683.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1683.png,COVID/images/COVID-1683.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1683.png,COVID/masks/COVID-1683.png,True,False,299,299,L,128.75,3b9cb95aa2f2c1abffe15e45555af339
COVID-1684.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1684.png,COVID/images/COVID-1684.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1684.png,COVID/masks/COVID-1684.png,True,False,299,299,L,151.59,0e37e76b484b7249044ecf46a3a2d9ba
COVID-1685.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1685.png,COVID/images/COVID-1685.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1685.png,COVID/masks/COVID-1685.png,True,False,299,299,L,154.78,9cfc64a8c7f35473deace9e5199dab91
COVID-1686.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1686.png,COVID/images/COVID-1686.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1686.png,COVID/masks/COVID-1686.png,True,False,299,299,L,165.03,2440fa05a3a85c6ea8cf2ba809dee90b
COVID-1687.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1687.png,COVID/images/COVID-1687.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1687.png,COVID/masks/COVID-1687.png,True,False,299,299,L,164.61,2dee1c42b75137cc77fe65a2b93b8e55
COVID-1688.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1688.png,COVID/images/COVID-1688.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1688.png,COVID/masks/COVID-1688.png,True,False,299,299,L,143.65,1096dbfa2a06d31135cd026e7b8fcf74
COVID-1689.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1689.png,COVID/images/COVID-1689.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1689.png,COVID/masks/COVID-1689.png,True,False,299,299,L,172.66,8d099171a7014167cf21abe5c5ed7d9b
COVID-169.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-169.png,COVID/images/COVID-169.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-169.png,COVID/masks/COVID-169.png,True,False,299,299,L,101.44,2b23c7c74424cecca1ee50c423d56e2d
COVID-1690.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1690.png,COVID/images/COVID-1690.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1690.png,COVID/masks/COVID-1690.png,True,False,299,299,L,144.35,c6a7d87b9480fa2ea25a69df1100eb1c
COVID-1691.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1691.png,COVID/images/COVID-1691.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1691.png,COVID/masks/COVID-1691.png,True,False,299,299,L,138.69,6c421ac81948003b42641cc633c16442
COVID-1692.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1692.png,COVID/images/COVID-1692.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1692.png,COVID/masks/COVID-1692.png,True,False,299,299,L,96.61,f3b1a934bbfcc12876fe9e04826ec476
COVID-1693.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1693.png,COVID/images/COVID-1693.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1693.png,COVID/masks/COVID-1693.png,True,False,299,299,L,155.41,9938042c70aee254c25fcc23a678ade9
COVID-1694.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1694.png,COVID/images/COVID-1694.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1694.png,COVID/masks/COVID-1694.png,True,False,299,299,L,146.29,fe2cc7cf129a47a6e0ea7153e714465b
COVID-1695.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1695.png,COVID/images/COVID-1695.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1695.png,COVID/masks/COVID-1695.png,True,False,299,299,L,159.93,7dee48ed959bf3d53462680170e5ba97
COVID-1696.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1696.png,COVID/images/COVID-1696.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1696.png,COVID/masks/COVID-1696.png,True,False,299,299,L,98.75,4c03ec004f80167fad328595b0075146
COVID-1697.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1697.png,COVID/images/COVID-1697.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1697.png,COVID/masks/COVID-1697.png,True,False,299,299,L,152.67,06e1da62dd2f4ef6673aa4fd11edf6f4
COVID-1698.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1698.png,COVID/images/COVID-1698.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1698.png,COVID/masks/COVID-1698.png,True,False,299,299,L,146.74,88874426192ef14fe829c3883b643436
COVID-1699.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1699.png,COVID/images/COVID-1699.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1699.png,COVID/masks/COVID-1699.png,True,False,299,299,L,164.58,d04437522bd46cdab58f404287a65613
COVID-17.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-17.png,COVID/images/COVID-17.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-17.png,COVID/masks/COVID-17.png,True,False,299,299,L,123.77,ad723892b73506714660bf9ae4bd8729
COVID-170.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-170.png,COVID/images/COVID-170.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-170.png,COVID/masks/COVID-170.png,True,False,299,299,L,167.91,b2e2b4b47756ca38e1fe75316bc81939
COVID-1700.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1700.png,COVID/images/COVID-1700.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1700.png,COVID/masks/COVID-1700.png,True,False,299,299,L,100.37,4f0d5fe5912d428234d6ecab2a2a7735
COVID-1701.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1701.png,COVID/images/COVID-1701.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1701.png,COVID/masks/COVID-1701.png,True,False,299,299,L,187.02,db198f265c5bd40a6ea31cf0345eb2da
COVID-1702.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1702.png,COVID/images/COVID-1702.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1702.png,COVID/masks/COVID-1702.png,True,False,299,299,L,100.04,781d760eb13950d36c2e2d48d19b6b1f
COVID-1703.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1703.png,COVID/images/COVID-1703.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1703.png,COVID/masks/COVID-1703.png,True,False,299,299,L,178.56,33fb4fd1039978aa4659e5bf43cb6ae6
COVID-1704.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1704.png,COVID/images/COVID-1704.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1704.png,COVID/masks/COVID-1704.png,True,False,299,299,L,139.63,ff1cd7c077a99c0b6a084396c7fc4f41
COVID-1705.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1705.png,COVID/images/COVID-1705.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1705.png,COVID/masks/COVID-1705.png,True,False,299,299,L,156.48,c2af6c90fb5f3059bd443422c42d4d98
COVID-1706.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1706.png,COVID/images/COVID-1706.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1706.png,COVID/masks/COVID-1706.png,True,False,299,299,L,103.67,42036a0c1edb3ae4fe357bbdc315edb0
COVID-1707.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1707.png,COVID/images/COVID-1707.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1707.png,COVID/masks/COVID-1707.png,True,False,299,299,L,135.68,fc6e5cc18d46c60d7377b6535a243e37
COVID-1708.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1708.png,COVID/images/COVID-1708.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1708.png,COVID/masks/COVID-1708.png,True,False,299,299,L,144.73,4e83a112d4bbb6db13bca96c2a7d5ad1
COVID-1709.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1709.png,COVID/images/COVID-1709.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1709.png,COVID/masks/COVID-1709.png,True,False,299,299,L,156.93,8bea53a4408aa2e108f667daf5adda7a
COVID-171.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-171.png,COVID/images/COVID-171.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-171.png,COVID/masks/COVID-171.png,True,False,299,299,L,147.79,ca7a6c829bb86f9dda9c7e072c691d31
COVID-1710.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1710.png,COVID/images/COVID-1710.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1710.png,COVID/masks/COVID-1710.png,True,False,299,299,L,101.28,7e33259eb79d9757a6be4faea3eec0cb
COVID-1711.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1711.png,COVID/images/COVID-1711.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1711.png,COVID/masks/COVID-1711.png,True,False,299,299,L,95.65,2219edac1f557f62c8cfa9d2da01c8f5
COVID-1712.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1712.png,COVID/images/COVID-1712.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1712.png,COVID/masks/COVID-1712.png,True,False,299,299,L,169.96,e1e4b5024128ddac45ba756869ab5afb
COVID-1713.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1713.png,COVID/images/COVID-1713.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1713.png,COVID/masks/COVID-1713.png,True,False,299,299,L,149.94,26e1a989a0a4d1439df760ba006f1f3a
COVID-1714.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1714.png,COVID/images/COVID-1714.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1714.png,COVID/masks/COVID-1714.png,True,False,299,299,L,103.88,9917e0d218357398ff50fb0c30df4341
COVID-1715.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1715.png,COVID/images/COVID-1715.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1715.png,COVID/masks/COVID-1715.png,True,False,299,299,L,150.57,17798803032da215d58bc06d1c5f6fe5
COVID-1716.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1716.png,COVID/images/COVID-1716.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1716.png,COVID/masks/COVID-1716.png,True,False,299,299,L,93.6,5d7a6bc179ced1cce20ed93c4f1a1eeb
COVID-1717.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1717.png,COVID/images/COVID-1717.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1717.png,COVID/masks/COVID-1717.png,True,False,299,299,L,89.0,92a199ff08b2317e9960b83dbc693267
COVID-1718.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1718.png,COVID/images/COVID-1718.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1718.png,COVID/masks/COVID-1718.png,True,False,299,299,L,144.07,4cfb97e587a89460256173e219ffd10a
COVID-1719.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1719.png,COVID/images/COVID-1719.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1719.png,COVID/masks/COVID-1719.png,True,False,299,299,L,143.22,7a197e2a613ad49111af39c8323b9362
COVID-172.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-172.png,COVID/images/COVID-172.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-172.png,COVID/masks/COVID-172.png,True,False,299,299,L,159.13,fc9bf801732b4c95e5910ad2d8af9d56
COVID-1720.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1720.png,COVID/images/COVID-1720.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1720.png,COVID/masks/COVID-1720.png,True,False,299,299,L,156.23,dda1c26f7599be49d80c68f45818503d
COVID-1721.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1721.png,COVID/images/COVID-1721.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1721.png,COVID/masks/COVID-1721.png,True,False,299,299,L,159.17,b98cb257c3b7174abec3bdffc750c354
COVID-1722.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1722.png,COVID/images/COVID-1722.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1722.png,COVID/masks/COVID-1722.png,True,False,299,299,L,169.58,0efacea780ef9150f36e2f773df1ab9d
COVID-1723.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1723.png,COVID/images/COVID-1723.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1723.png,COVID/masks/COVID-1723.png,True,False,299,299,L,144.89,07dbdf9514c252ca4652234f80c65db5
COVID-1724.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1724.png,COVID/images/COVID-1724.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1724.png,COVID/masks/COVID-1724.png,True,False,299,299,L,156.87,9898c3efb56de2262d16577d5c628195
COVID-1725.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1725.png,COVID/images/COVID-1725.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1725.png,COVID/masks/COVID-1725.png,True,False,299,299,L,138.1,8956fd0ab5c72a340df3103b8f83d39c
COVID-1726.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1726.png,COVID/images/COVID-1726.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1726.png,COVID/masks/COVID-1726.png,True,False,299,299,L,135.77,4833744580afc09390bcf0533eca9e2f
COVID-1727.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1727.png,COVID/images/COVID-1727.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1727.png,COVID/masks/COVID-1727.png,True,False,299,299,L,169.9,8b2c7d31ed90c6a1953e0c4388e40a64
COVID-1728.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1728.png,COVID/images/COVID-1728.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1728.png,COVID/masks/COVID-1728.png,True,False,299,299,L,125.2,4058151def8898dbb3e33d46b36b3e0d
COVID-1729.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1729.png,COVID/images/COVID-1729.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1729.png,COVID/masks/COVID-1729.png,True,False,299,299,L,137.39,85483548e24efa9405cd2fed476b3724
COVID-173.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-173.png,COVID/images/COVID-173.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-173.png,COVID/masks/COVID-173.png,True,False,299,299,L,158.72,ba5dd9a06ddeec3ba77b695fc71bf81b
COVID-1730.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1730.png,COVID/images/COVID-1730.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1730.png,COVID/masks/COVID-1730.png,True,False,299,299,L,124.31,c7dbea060fe62dcc46e0bf913ab9582a
COVID-1731.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1731.png,COVID/images/COVID-1731.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1731.png,COVID/masks/COVID-1731.png,True,False,299,299,L,160.42,3e069c3ded19b86bc134796accbbf306
COVID-1732.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1732.png,COVID/images/COVID-1732.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1732.png,COVID/masks/COVID-1732.png,True,False,299,299,L,155.83,08d1fa3f89554b354d2b9df615830033
COVID-1733.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1733.png,COVID/images/COVID-1733.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1733.png,COVID/masks/COVID-1733.png,True,False,299,299,L,167.56,aa4635fc36cc3557664557d211f98fa5
COVID-1734.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1734.png,COVID/images/COVID-1734.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1734.png,COVID/masks/COVID-1734.png,True,False,299,299,L,153.95,f146d69faa041c9023999d1a5f97c81f
COVID-1735.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1735.png,COVID/images/COVID-1735.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1735.png,COVID/masks/COVID-1735.png,True,False,299,299,L,150.4,3bbb4fac9c3404dcdb9f62d75fa44589
COVID-1736.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1736.png,COVID/images/COVID-1736.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1736.png,COVID/masks/COVID-1736.png,True,False,299,299,L,156.5,dbb176629f1b5bdfb1d3d3bbade10283
COVID-1737.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1737.png,COVID/images/COVID-1737.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1737.png,COVID/masks/COVID-1737.png,True,False,299,299,L,144.95,e2ae74b6daf422c84389875caaec8486
COVID-1738.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1738.png,COVID/images/COVID-1738.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1738.png,COVID/masks/COVID-1738.png,True,False,299,299,L,136.24,8e0f7c3b38526381f59377d2dea43cea
COVID-1739.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1739.png,COVID/images/COVID-1739.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1739.png,COVID/masks/COVID-1739.png,True,False,299,299,L,171.88,5d5f11754c907b24ab4e25f82613e881
COVID-174.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-174.png,COVID/images/COVID-174.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-174.png,COVID/masks/COVID-174.png,True,False,299,299,L,147.7,e7ced2dd33cab7aad29dea0a279b5eea
COVID-1740.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1740.png,COVID/images/COVID-1740.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1740.png,COVID/masks/COVID-1740.png,True,False,299,299,L,129.93,f0a4ce699f689a6d4dad59f316fa5947
COVID-1741.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1741.png,COVID/images/COVID-1741.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1741.png,COVID/masks/COVID-1741.png,True,False,299,299,L,158.09,ffb210f91f5d543b808294b2e4c10173
COVID-1742.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1742.png,COVID/images/COVID-1742.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1742.png,COVID/masks/COVID-1742.png,True,False,299,299,L,129.01,24fe1fca23dd976b9b166294e8967589
COVID-1743.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1743.png,COVID/images/COVID-1743.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1743.png,COVID/masks/COVID-1743.png,True,False,299,299,L,139.29,82fcc7d0eaffebaa9ae710e2a5400a28
COVID-1744.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1744.png,COVID/images/COVID-1744.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1744.png,COVID/masks/COVID-1744.png,True,False,299,299,L,135.6,57c0a22782b8c0d0626d53783bf78fb5
COVID-1745.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1745.png,COVID/images/COVID-1745.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1745.png,COVID/masks/COVID-1745.png,True,False,299,299,L,142.74,62dfc808829b8f8159a8514df07ab760
COVID-1746.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1746.png,COVID/images/COVID-1746.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1746.png,COVID/masks/COVID-1746.png,True,False,299,299,L,147.48,feb338d408721c95654ade6b9331fed7
COVID-1747.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1747.png,COVID/images/COVID-1747.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1747.png,COVID/masks/COVID-1747.png,True,False,299,299,L,125.59,818e58e7e635b204929482c20b3f9971
COVID-1748.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1748.png,COVID/images/COVID-1748.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1748.png,COVID/masks/COVID-1748.png,True,False,299,299,L,144.76,56a2b404cf3196fb88f6a80f451cd52a
COVID-1749.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1749.png,COVID/images/COVID-1749.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1749.png,COVID/masks/COVID-1749.png,True,False,299,299,L,139.22,b1d0be833cce6f347a64f7c08d0a924a
COVID-175.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-175.png,COVID/images/COVID-175.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-175.png,COVID/masks/COVID-175.png,True,False,299,299,L,149.85,ee96eed714ccb5b8e3a7f39d2b76407e
COVID-1750.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1750.png,COVID/images/COVID-1750.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1750.png,COVID/masks/COVID-1750.png,True,False,299,299,L,121.21,df2548c8569541417356658772fc3a57
COVID-1751.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1751.png,COVID/images/COVID-1751.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1751.png,COVID/masks/COVID-1751.png,True,False,299,299,L,125.0,3a96f9c90338ad58d3a21d6af98cb21f
COVID-1752.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1752.png,COVID/images/COVID-1752.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1752.png,COVID/masks/COVID-1752.png,True,False,299,299,L,142.73,e3a8fc915e86708379c5ec9bdce40e8c
COVID-1753.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1753.png,COVID/images/COVID-1753.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1753.png,COVID/masks/COVID-1753.png,True,False,299,299,L,142.1,eb1d4a8afbc8dcc872536837c032f63c
COVID-1754.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1754.png,COVID/images/COVID-1754.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1754.png,COVID/masks/COVID-1754.png,True,False,299,299,L,157.87,58796026b8d80ece25c094bc4e656a94
COVID-1755.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1755.png,COVID/images/COVID-1755.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1755.png,COVID/masks/COVID-1755.png,True,False,299,299,L,141.06,3b53c9de066be55f6b40c436a21a2f8f
COVID-1756.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1756.png,COVID/images/COVID-1756.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1756.png,COVID/masks/COVID-1756.png,True,False,299,299,L,147.01,5683227dc42dbdb9b9e07eeee2a18a56
COVID-1757.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1757.png,COVID/images/COVID-1757.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1757.png,COVID/masks/COVID-1757.png,True,False,299,299,L,42.05,1de049039272ffd58be1a403a46a725c
COVID-1758.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1758.png,COVID/images/COVID-1758.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1758.png,COVID/masks/COVID-1758.png,True,False,299,299,L,156.37,e8c1b760a806ef2cee36b37d2c8c9156
COVID-1759.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1759.png,COVID/images/COVID-1759.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1759.png,COVID/masks/COVID-1759.png,True,False,299,299,L,138.46,1ffa887da6d43fcd8b2c3b3d772887df
COVID-176.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-176.png,COVID/images/COVID-176.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-176.png,COVID/masks/COVID-176.png,True,False,299,299,L,165.28,701fa3e5a51970d87271e029526d0d56
COVID-1760.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1760.png,COVID/images/COVID-1760.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1760.png,COVID/masks/COVID-1760.png,True,False,299,299,L,97.15,17aa49e4672bca15226623e669e9d16f
COVID-1761.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1761.png,COVID/images/COVID-1761.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1761.png,COVID/masks/COVID-1761.png,True,False,299,299,L,124.67,a9a47d01b7c6b4811c39d61765ebf3ce
COVID-1762.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1762.png,COVID/images/COVID-1762.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1762.png,COVID/masks/COVID-1762.png,True,False,299,299,L,155.86,d2dcc5db752d17bfd48914a6b3694d02
COVID-1763.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1763.png,COVID/images/COVID-1763.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1763.png,COVID/masks/COVID-1763.png,True,False,299,299,L,141.1,1da1909a7ac7da4c905c10a021b73496
COVID-1764.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1764.png,COVID/images/COVID-1764.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1764.png,COVID/masks/COVID-1764.png,True,False,299,299,L,132.11,8a74eb60bf0d1291eb178c0b396a671b
COVID-1765.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1765.png,COVID/images/COVID-1765.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1765.png,COVID/masks/COVID-1765.png,True,False,299,299,L,147.39,14c86437d7228722260dab8bea2ce2a2
COVID-1766.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1766.png,COVID/images/COVID-1766.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1766.png,COVID/masks/COVID-1766.png,True,False,299,299,L,159.54,00ea43dae61d135330f0ca8c9254169d
COVID-1767.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1767.png,COVID/images/COVID-1767.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1767.png,COVID/masks/COVID-1767.png,True,False,299,299,L,169.41,ad6e7be4e3613f8b99adb3abac4b867a
COVID-1768.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1768.png,COVID/images/COVID-1768.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1768.png,COVID/masks/COVID-1768.png,True,False,299,299,L,161.87,bcab43a387a40795d52c6bd402475ee4
COVID-1769.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1769.png,COVID/images/COVID-1769.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1769.png,COVID/masks/COVID-1769.png,True,False,299,299,L,180.97,21edd226a21370104fd877a76e34328b
COVID-177.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-177.png,COVID/images/COVID-177.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-177.png,COVID/masks/COVID-177.png,True,False,299,299,L,152.06,b580780f52a370eb2ff5514616093b1d
COVID-1770.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1770.png,COVID/images/COVID-1770.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1770.png,COVID/masks/COVID-1770.png,True,False,299,299,L,139.66,1fee193dc47317a3d66859ee18b85a0c
COVID-1771.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1771.png,COVID/images/COVID-1771.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1771.png,COVID/masks/COVID-1771.png,True,False,299,299,L,155.85,364969aaa42f0195eb41e66146ec9f13
COVID-1772.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1772.png,COVID/images/COVID-1772.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1772.png,COVID/masks/COVID-1772.png,True,False,299,299,L,154.81,bc82d656ed3ece91d10ab05ddfe11217
COVID-1773.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1773.png,COVID/images/COVID-1773.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1773.png,COVID/masks/COVID-1773.png,True,False,299,299,L,178.12,4e3c75fc928387ac0e241951c184873b
COVID-1774.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1774.png,COVID/images/COVID-1774.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1774.png,COVID/masks/COVID-1774.png,True,False,299,299,L,151.91,fe876f54030ba3aaf0d2c39518a9d143
COVID-1775.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1775.png,COVID/images/COVID-1775.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1775.png,COVID/masks/COVID-1775.png,True,False,299,299,L,172.1,830a0b6c7c7f9020acce7debe3580331
COVID-1776.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1776.png,COVID/images/COVID-1776.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1776.png,COVID/masks/COVID-1776.png,True,False,299,299,L,99.97,82c7dec04ecec09dc301940234809c2c
COVID-1777.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1777.png,COVID/images/COVID-1777.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1777.png,COVID/masks/COVID-1777.png,True,False,299,299,L,154.14,0a8607acd693a46908ea65c9e4b3d8b8
COVID-1778.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1778.png,COVID/images/COVID-1778.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1778.png,COVID/masks/COVID-1778.png,True,False,299,299,L,157.03,3ea54492c673012ec8d79e51f66943f5
COVID-1779.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1779.png,COVID/images/COVID-1779.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1779.png,COVID/masks/COVID-1779.png,True,False,299,299,L,144.53,43cbae3f8fcb05135b530ea5ed7330cf
COVID-178.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-178.png,COVID/images/COVID-178.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-178.png,COVID/masks/COVID-178.png,True,False,299,299,L,132.57,c7d18c7f5514fad074d936a43b74a913
COVID-1780.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1780.png,COVID/images/COVID-1780.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1780.png,COVID/masks/COVID-1780.png,True,False,299,299,L,165.02,92497104c9f9ee9ee877ce39c33b5ca0
COVID-1781.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1781.png,COVID/images/COVID-1781.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1781.png,COVID/masks/COVID-1781.png,True,False,299,299,L,144.34,5f36189a0dfb330ddfb9b73f12de58c8
COVID-1782.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1782.png,COVID/images/COVID-1782.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1782.png,COVID/masks/COVID-1782.png,True,False,299,299,L,170.89,316c3f9b2c706e9df7ec2b649c7d4aa9
COVID-1783.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1783.png,COVID/images/COVID-1783.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1783.png,COVID/masks/COVID-1783.png,True,False,299,299,L,65.12,5a50a62cf6cc894677c0716c86163c1e
COVID-1784.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1784.png,COVID/images/COVID-1784.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1784.png,COVID/masks/COVID-1784.png,True,False,299,299,L,160.58,05e2611079547972200c4b473d5b8c8c
COVID-1785.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1785.png,COVID/images/COVID-1785.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1785.png,COVID/masks/COVID-1785.png,True,False,299,299,L,120.61,d83b5c4b739ab515d1008fb57b077656
COVID-1786.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1786.png,COVID/images/COVID-1786.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1786.png,COVID/masks/COVID-1786.png,True,False,299,299,L,140.26,9a4f0bd661744d9bfda926bdaf400a0c
COVID-1787.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1787.png,COVID/images/COVID-1787.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1787.png,COVID/masks/COVID-1787.png,True,False,299,299,L,181.21,eeb81ba7c4f4d2b803f0b60fd3d9b740
COVID-1788.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1788.png,COVID/images/COVID-1788.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1788.png,COVID/masks/COVID-1788.png,True,False,299,299,L,120.78,9c872aed5714fa08cf70d143d0c599f4
COVID-1789.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1789.png,COVID/images/COVID-1789.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1789.png,COVID/masks/COVID-1789.png,True,False,299,299,L,173.19,f4fe2f04fb2afbb1ca42f8a1ca4c8a78
COVID-179.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-179.png,COVID/images/COVID-179.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-179.png,COVID/masks/COVID-179.png,True,False,299,299,L,102.68,b0fc7a7dc66ff0e22a6f163012726364
COVID-1790.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1790.png,COVID/images/COVID-1790.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1790.png,COVID/masks/COVID-1790.png,True,False,299,299,L,120.57,d108ed92e7a38f8a5012b05755164d56
COVID-1791.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1791.png,COVID/images/COVID-1791.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1791.png,COVID/masks/COVID-1791.png,True,False,299,299,L,175.87,5b94711427154172181faadf921b7be0
COVID-1792.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1792.png,COVID/images/COVID-1792.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1792.png,COVID/masks/COVID-1792.png,True,False,299,299,L,180.12,ee13d18361387f88069119b88416869d
COVID-1793.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1793.png,COVID/images/COVID-1793.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1793.png,COVID/masks/COVID-1793.png,True,False,299,299,L,170.43,c0b2b7142b2356a0be6db4f0d58a47cd
COVID-1794.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1794.png,COVID/images/COVID-1794.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1794.png,COVID/masks/COVID-1794.png,True,False,299,299,L,151.46,4700a39e25585c7dd6cb49921459875a
COVID-1795.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1795.png,COVID/images/COVID-1795.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1795.png,COVID/masks/COVID-1795.png,True,False,299,299,L,146.92,4e196a7a9f2aaa76274eb019ece61b5c
COVID-1796.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1796.png,COVID/images/COVID-1796.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1796.png,COVID/masks/COVID-1796.png,True,False,299,299,L,185.7,5b92eac17fda0e3d7aa9da4833172b0a
COVID-1797.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1797.png,COVID/images/COVID-1797.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1797.png,COVID/masks/COVID-1797.png,True,False,299,299,L,90.35,133a92d25c9480a3761b1ce21c1b39d5
COVID-1798.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1798.png,COVID/images/COVID-1798.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1798.png,COVID/masks/COVID-1798.png,True,False,299,299,L,131.0,63957d035a4b5e7a8121aeec4f4eda73
COVID-1799.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1799.png,COVID/images/COVID-1799.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1799.png,COVID/masks/COVID-1799.png,True,False,299,299,L,93.81,3f2831c748be9bc9682b278780944404
COVID-18.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-18.png,COVID/images/COVID-18.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-18.png,COVID/masks/COVID-18.png,True,False,299,299,L,151.38,a00f1e48eaebd032eee090def7795dac
COVID-180.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-180.png,COVID/images/COVID-180.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-180.png,COVID/masks/COVID-180.png,True,False,299,299,L,134.01,d9770d962a24897f6013f01866b16761
COVID-1800.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1800.png,COVID/images/COVID-1800.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1800.png,COVID/masks/COVID-1800.png,True,False,299,299,L,154.03,f986bc2bfde4c8469c96e190142ca2ba
COVID-1801.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1801.png,COVID/images/COVID-1801.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1801.png,COVID/masks/COVID-1801.png,True,False,299,299,L,157.18,96d02325e9746ce316be199af69985c9
COVID-1802.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1802.png,COVID/images/COVID-1802.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1802.png,COVID/masks/COVID-1802.png,True,False,299,299,L,148.06,68f1d64916baf6f0a1105417d625adf2
COVID-1803.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1803.png,COVID/images/COVID-1803.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1803.png,COVID/masks/COVID-1803.png,True,False,299,299,L,154.69,1460224fdc62cdd2e1c6f841d49b2222
COVID-1804.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1804.png,COVID/images/COVID-1804.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1804.png,COVID/masks/COVID-1804.png,True,False,299,299,L,144.92,ba34d8123e2690d8e299b1a2c8495d61
COVID-1805.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1805.png,COVID/images/COVID-1805.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1805.png,COVID/masks/COVID-1805.png,True,False,299,299,L,178.61,b752ee57c76deae0bd196c48e9700983
COVID-1806.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1806.png,COVID/images/COVID-1806.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1806.png,COVID/masks/COVID-1806.png,True,False,299,299,L,149.43,efe8dc8ab2697687e10d916796a545e1
COVID-1807.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1807.png,COVID/images/COVID-1807.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1807.png,COVID/masks/COVID-1807.png,True,False,299,299,L,171.55,73918eedb0e13acc8799183bb31a1c54
COVID-1808.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1808.png,COVID/images/COVID-1808.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1808.png,COVID/masks/COVID-1808.png,True,False,299,299,L,163.41,6e6c343c3e0c067cb13615b7aa839b00
COVID-1809.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1809.png,COVID/images/COVID-1809.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1809.png,COVID/masks/COVID-1809.png,True,False,299,299,L,176.91,2c49be3f04c5f67f466a2f1e8bcaafb8
COVID-181.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-181.png,COVID/images/COVID-181.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-181.png,COVID/masks/COVID-181.png,True,False,299,299,L,117.79,452e5741b587b2c648d2dfa47a9fe728
COVID-1810.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1810.png,COVID/images/COVID-1810.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1810.png,COVID/masks/COVID-1810.png,True,False,299,299,L,143.41,522c4798971f4115c1a323914d0760b5
COVID-1811.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1811.png,COVID/images/COVID-1811.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1811.png,COVID/masks/COVID-1811.png,True,False,299,299,L,136.76,1809ba2cfb0688c142ba7529d3fde908
COVID-1812.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1812.png,COVID/images/COVID-1812.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1812.png,COVID/masks/COVID-1812.png,True,False,299,299,L,142.23,9f1292f526cd4b65ea4572461940f101
COVID-1813.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1813.png,COVID/images/COVID-1813.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1813.png,COVID/masks/COVID-1813.png,True,False,299,299,L,146.59,efb3f95fca78b35f36dad8c9ba67edb7
COVID-1814.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1814.png,COVID/images/COVID-1814.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1814.png,COVID/masks/COVID-1814.png,True,False,299,299,L,168.53,ea4c6326e4d2af4210d67ad03d76a4b7
COVID-1815.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1815.png,COVID/images/COVID-1815.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1815.png,COVID/masks/COVID-1815.png,True,False,299,299,L,145.63,2cbb5aa2ed2576735ae8f15aa7487c26
COVID-1816.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1816.png,COVID/images/COVID-1816.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1816.png,COVID/masks/COVID-1816.png,True,False,299,299,L,169.05,e166f3b311f044070e9596035c47a907
COVID-1817.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1817.png,COVID/images/COVID-1817.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1817.png,COVID/masks/COVID-1817.png,True,False,299,299,L,143.44,0d9c2549f6f5482bfd0f33bf5e694cce
COVID-1818.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1818.png,COVID/images/COVID-1818.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1818.png,COVID/masks/COVID-1818.png,True,False,299,299,L,157.86,2cae0bdad731c00acd15096c25e20805
COVID-1819.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1819.png,COVID/images/COVID-1819.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1819.png,COVID/masks/COVID-1819.png,True,False,299,299,L,153.48,f3fcdecdd0cf569394a395838269ae65
COVID-182.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-182.png,COVID/images/COVID-182.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-182.png,COVID/masks/COVID-182.png,True,False,299,299,L,174.23,a19da82a99a2897bc29d46c24eb7811b
COVID-1820.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1820.png,COVID/images/COVID-1820.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1820.png,COVID/masks/COVID-1820.png,True,False,299,299,L,182.86,1ac8c28447430ebbc3bc6f5324709650
COVID-1821.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1821.png,COVID/images/COVID-1821.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1821.png,COVID/masks/COVID-1821.png,True,False,299,299,L,133.45,908094a4cdf60b72abc82682cd51cd67
COVID-1822.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1822.png,COVID/images/COVID-1822.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1822.png,COVID/masks/COVID-1822.png,True,False,299,299,L,120.12,117c9e761e1cedfce21a8bd0eb088a9f
COVID-1823.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1823.png,COVID/images/COVID-1823.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1823.png,COVID/masks/COVID-1823.png,True,False,299,299,L,114.06,b3f97f9a7db618aa6902c7f2f9b6f805
COVID-1824.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1824.png,COVID/images/COVID-1824.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1824.png,COVID/masks/COVID-1824.png,True,False,299,299,L,182.43,4367658d0d05b21fa4582910bceb7e8f
COVID-1825.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1825.png,COVID/images/COVID-1825.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1825.png,COVID/masks/COVID-1825.png,True,False,299,299,L,110.96,83ae784b23a96b8e70cf084e8d4c8d9c
COVID-1826.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1826.png,COVID/images/COVID-1826.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1826.png,COVID/masks/COVID-1826.png,True,False,299,299,L,139.06,7ed05a5f12a7584d3d1de489b8a7fae3
COVID-1827.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1827.png,COVID/images/COVID-1827.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1827.png,COVID/masks/COVID-1827.png,True,False,299,299,L,174.49,642b090391898ace27ec313aca10e7df
COVID-1828.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1828.png,COVID/images/COVID-1828.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1828.png,COVID/masks/COVID-1828.png,True,False,299,299,L,137.39,36cb6a596968d2c0e78f6e784da835f1
COVID-1829.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1829.png,COVID/images/COVID-1829.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1829.png,COVID/masks/COVID-1829.png,True,False,299,299,L,148.4,81d983fc550dce74e45c2b515e11bed2
COVID-183.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-183.png,COVID/images/COVID-183.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-183.png,COVID/masks/COVID-183.png,True,False,299,299,L,161.57,d96bba153d71d4f996711f1603b1fe39
COVID-1830.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1830.png,COVID/images/COVID-1830.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1830.png,COVID/masks/COVID-1830.png,True,False,299,299,L,152.23,15cdbf1fba5e74dcbfdda70668700c8e
COVID-1831.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1831.png,COVID/images/COVID-1831.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1831.png,COVID/masks/COVID-1831.png,True,False,299,299,L,106.89,5ab7c0180ec5e270c69ef3a531e6828a
COVID-1832.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1832.png,COVID/images/COVID-1832.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1832.png,COVID/masks/COVID-1832.png,True,False,299,299,L,152.94,25530f4c1710215b655c705de51bbcbe
COVID-1833.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1833.png,COVID/images/COVID-1833.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1833.png,COVID/masks/COVID-1833.png,True,False,299,299,L,129.52,c3a093a4dc1d92a24a5241916a89bc6e
COVID-1834.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1834.png,COVID/images/COVID-1834.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1834.png,COVID/masks/COVID-1834.png,True,False,299,299,L,154.48,72a7b73ee25b692a5ab188a94439cc0c
COVID-1835.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1835.png,COVID/images/COVID-1835.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1835.png,COVID/masks/COVID-1835.png,True,False,299,299,L,138.41,8a56307e4e5bb0afbda66ed240b0792f
COVID-1836.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1836.png,COVID/images/COVID-1836.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1836.png,COVID/masks/COVID-1836.png,True,False,299,299,L,185.03,cf0c8c24e2b938f65dfcc0b4de45515f
COVID-1837.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1837.png,COVID/images/COVID-1837.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1837.png,COVID/masks/COVID-1837.png,True,False,299,299,L,120.4,97dbe1a453abb2eb69815618fb4b5646
COVID-1838.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1838.png,COVID/images/COVID-1838.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1838.png,COVID/masks/COVID-1838.png,True,False,299,299,L,174.77,a360b1c82a713fcac3e6243faf55690d
COVID-1839.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1839.png,COVID/images/COVID-1839.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1839.png,COVID/masks/COVID-1839.png,True,False,299,299,L,140.46,97d9f47f73d7e882cc8e930126edf2ca
COVID-184.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-184.png,COVID/images/COVID-184.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-184.png,COVID/masks/COVID-184.png,True,False,299,299,L,113.34,b97c9dacf8908dd4a5b7114c7d30f156
COVID-1840.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1840.png,COVID/images/COVID-1840.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1840.png,COVID/masks/COVID-1840.png,True,False,299,299,L,158.21,08a0b8417c94e89a7b51196b40efba34
COVID-1841.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1841.png,COVID/images/COVID-1841.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1841.png,COVID/masks/COVID-1841.png,True,False,299,299,L,111.33,3819ab370fd240e15985b0c691ba15e2
COVID-1842.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1842.png,COVID/images/COVID-1842.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1842.png,COVID/masks/COVID-1842.png,True,False,299,299,L,115.64,9ad9a92d98a26fcbd191921e297e2cbd
COVID-1843.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1843.png,COVID/images/COVID-1843.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1843.png,COVID/masks/COVID-1843.png,True,False,299,299,L,169.47,b55fc54ad8bcca494fbddfa577363590
COVID-1844.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1844.png,COVID/images/COVID-1844.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1844.png,COVID/masks/COVID-1844.png,True,False,299,299,L,167.07,eeb8c764098d49f1b938710a528a3bb4
COVID-1845.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1845.png,COVID/images/COVID-1845.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1845.png,COVID/masks/COVID-1845.png,True,False,299,299,L,144.32,603665ce99bd267189893185a27caa8d
COVID-1846.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1846.png,COVID/images/COVID-1846.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1846.png,COVID/masks/COVID-1846.png,True,False,299,299,L,176.86,a75176162aa408cff7aa04bd7bc383ef
COVID-1847.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1847.png,COVID/images/COVID-1847.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1847.png,COVID/masks/COVID-1847.png,True,False,299,299,L,115.53,0d9e61bc8e181057110014e5c4eb1010
COVID-1848.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1848.png,COVID/images/COVID-1848.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1848.png,COVID/masks/COVID-1848.png,True,False,299,299,L,117.53,56cc22e553fc7deb21c96e807cbaae2c
COVID-1849.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1849.png,COVID/images/COVID-1849.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1849.png,COVID/masks/COVID-1849.png,True,False,299,299,L,141.75,387cfff46916c683afb4c4ec6a9746ef
COVID-185.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-185.png,COVID/images/COVID-185.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-185.png,COVID/masks/COVID-185.png,True,False,299,299,L,152.79,5979765fb69d40315f4591f415b08096
COVID-1850.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1850.png,COVID/images/COVID-1850.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1850.png,COVID/masks/COVID-1850.png,True,False,299,299,L,119.44,9e2d746479c68e8f2b8b1c12f127de28
COVID-1851.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1851.png,COVID/images/COVID-1851.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1851.png,COVID/masks/COVID-1851.png,True,False,299,299,L,174.81,9929840bc7ac9c7ce1b0bfa22a8f27e2
COVID-1852.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1852.png,COVID/images/COVID-1852.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1852.png,COVID/masks/COVID-1852.png,True,False,299,299,L,155.75,b2c3d552db8d0cc00cfd99e8cd630557
COVID-1853.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1853.png,COVID/images/COVID-1853.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1853.png,COVID/masks/COVID-1853.png,True,False,299,299,L,185.33,eab431a6bb0a4e96ce44aa5bf576dddf
COVID-1854.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1854.png,COVID/images/COVID-1854.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1854.png,COVID/masks/COVID-1854.png,True,False,299,299,L,138.86,a868c36bf955c5d9c85ac62704f27faa
COVID-1855.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1855.png,COVID/images/COVID-1855.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1855.png,COVID/masks/COVID-1855.png,True,False,299,299,L,158.86,32be51b6583870a9de1eb96418abe36d
COVID-1856.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1856.png,COVID/images/COVID-1856.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1856.png,COVID/masks/COVID-1856.png,True,False,299,299,L,155.26,d0a02b7f97587626c2109a7fa86dc20e
COVID-1857.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1857.png,COVID/images/COVID-1857.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1857.png,COVID/masks/COVID-1857.png,True,False,299,299,L,136.88,c1ec68048e9c38c99e8be70d6a2fa6c4
COVID-1858.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1858.png,COVID/images/COVID-1858.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1858.png,COVID/masks/COVID-1858.png,True,False,299,299,L,163.55,706500e606d18e345186145f72de2181
COVID-1859.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1859.png,COVID/images/COVID-1859.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1859.png,COVID/masks/COVID-1859.png,True,False,299,299,L,120.84,d6963fc8208a1342413f5ed6bd746bd8
COVID-186.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-186.png,COVID/images/COVID-186.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-186.png,COVID/masks/COVID-186.png,True,False,299,299,L,133.31,4288dae5c9eb9abb5ff33668ea206bd3
COVID-1860.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1860.png,COVID/images/COVID-1860.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1860.png,COVID/masks/COVID-1860.png,True,False,299,299,L,160.69,1b0839e4c94d06e401a7b95a53b2e375
COVID-1861.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1861.png,COVID/images/COVID-1861.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1861.png,COVID/masks/COVID-1861.png,True,False,299,299,L,137.92,4799e19b196d2f2c2dc709effe4367cc
COVID-1862.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1862.png,COVID/images/COVID-1862.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1862.png,COVID/masks/COVID-1862.png,True,False,299,299,L,95.09,c4eb3ff6394986148ef71eaea946cc44
COVID-1863.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1863.png,COVID/images/COVID-1863.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1863.png,COVID/masks/COVID-1863.png,True,False,299,299,L,147.94,e4a8a40a78e493f8c2812f974256fa85
COVID-1864.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1864.png,COVID/images/COVID-1864.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1864.png,COVID/masks/COVID-1864.png,True,False,299,299,L,114.92,9691f1c3c1efa71758f6abeb4cc398ae
COVID-1865.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1865.png,COVID/images/COVID-1865.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1865.png,COVID/masks/COVID-1865.png,True,False,299,299,L,113.23,a78820f600a8d749afa141ee71e82a5c
COVID-1866.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1866.png,COVID/images/COVID-1866.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1866.png,COVID/masks/COVID-1866.png,True,False,299,299,L,118.09,60ee741aca1a31bdfe583aae4eb3d2b1
COVID-1867.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1867.png,COVID/images/COVID-1867.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1867.png,COVID/masks/COVID-1867.png,True,False,299,299,L,118.46,a5b0e4cc5ee061ea8ceb924f6eec90ed
COVID-1868.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1868.png,COVID/images/COVID-1868.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1868.png,COVID/masks/COVID-1868.png,True,False,299,299,L,140.37,129cdf80690873e7f0ab974e4e1dddbe
COVID-1869.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1869.png,COVID/images/COVID-1869.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1869.png,COVID/masks/COVID-1869.png,True,False,299,299,L,149.18,b31186c65e867504eef41f8a57e04216
COVID-187.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-187.png,COVID/images/COVID-187.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-187.png,COVID/masks/COVID-187.png,True,False,299,299,L,91.31,4d439614b18763d1059e9e99dbbda79a
COVID-1870.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1870.png,COVID/images/COVID-1870.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1870.png,COVID/masks/COVID-1870.png,True,False,299,299,L,165.58,958143ee32ac4cec866b4e7680430231
COVID-1871.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1871.png,COVID/images/COVID-1871.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1871.png,COVID/masks/COVID-1871.png,True,False,299,299,L,129.54,bc8aac7de39691fe53a05a6bb5afbf7c
COVID-1872.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1872.png,COVID/images/COVID-1872.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1872.png,COVID/masks/COVID-1872.png,True,False,299,299,L,123.67,9cf1d90495d693fc011dd0e54f4e8f53
COVID-1873.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1873.png,COVID/images/COVID-1873.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1873.png,COVID/masks/COVID-1873.png,True,False,299,299,L,134.2,ee5ffd4e9153c5d2a4108f95087d51ef
COVID-1874.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1874.png,COVID/images/COVID-1874.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1874.png,COVID/masks/COVID-1874.png,True,False,299,299,L,86.35,287c7b90590e8ba31d1785b50ad60258
COVID-1875.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1875.png,COVID/images/COVID-1875.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1875.png,COVID/masks/COVID-1875.png,True,False,299,299,L,141.21,dcb460d642d1f780076b79bc564f3e28
COVID-1876.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1876.png,COVID/images/COVID-1876.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1876.png,COVID/masks/COVID-1876.png,True,False,299,299,L,25.72,9de0cb1c021fe0d4fe4e4ba5ef87caa6
COVID-1877.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1877.png,COVID/images/COVID-1877.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1877.png,COVID/masks/COVID-1877.png,True,False,299,299,L,127.44,961743c6ceee6af3f48de984c7273df4
COVID-1878.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1878.png,COVID/images/COVID-1878.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1878.png,COVID/masks/COVID-1878.png,True,False,299,299,L,188.07,6386f5740115429ba496c4fa44240119
COVID-1879.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1879.png,COVID/images/COVID-1879.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1879.png,COVID/masks/COVID-1879.png,True,False,299,299,L,147.5,36ab3ccf684dc767540df586772918e9
COVID-188.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-188.png,COVID/images/COVID-188.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-188.png,COVID/masks/COVID-188.png,True,False,299,299,L,163.72,af718fa998dd57c264df50f85ec01517
COVID-1880.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1880.png,COVID/images/COVID-1880.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1880.png,COVID/masks/COVID-1880.png,True,False,299,299,L,137.03,e5af38106698f96e70e958ed4601ad81
COVID-1881.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1881.png,COVID/images/COVID-1881.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1881.png,COVID/masks/COVID-1881.png,True,False,299,299,L,174.76,26d48323430fb1ee01f72e2a2195f2ea
COVID-1882.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1882.png,COVID/images/COVID-1882.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1882.png,COVID/masks/COVID-1882.png,True,False,299,299,L,159.37,a1f0adfdcccd79dbe68b1395736cb2ed
COVID-1883.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1883.png,COVID/images/COVID-1883.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1883.png,COVID/masks/COVID-1883.png,True,False,299,299,L,110.75,862ebca7ea6206f750868522593d8afd
COVID-1884.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1884.png,COVID/images/COVID-1884.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1884.png,COVID/masks/COVID-1884.png,True,False,299,299,L,187.62,8c71773ff05aa49b4e316a3740397317
COVID-1885.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1885.png,COVID/images/COVID-1885.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1885.png,COVID/masks/COVID-1885.png,True,False,299,299,L,153.31,98092700d78c2f35c8c8fc37abbafb40
COVID-1886.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1886.png,COVID/images/COVID-1886.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1886.png,COVID/masks/COVID-1886.png,True,False,299,299,L,128.27,1f47dbb84239c54fca21dac1a6d8cb80
COVID-1887.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1887.png,COVID/images/COVID-1887.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1887.png,COVID/masks/COVID-1887.png,True,False,299,299,L,107.2,fd407e215d7a661027a200a40af9d146
COVID-1888.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1888.png,COVID/images/COVID-1888.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1888.png,COVID/masks/COVID-1888.png,True,False,299,299,L,153.85,10be219a650ced3741ac566c82f3cb8a
COVID-1889.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1889.png,COVID/images/COVID-1889.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1889.png,COVID/masks/COVID-1889.png,True,False,299,299,L,142.91,1a8d012f15bec92de43dff80fa981ed4
COVID-189.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-189.png,COVID/images/COVID-189.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-189.png,COVID/masks/COVID-189.png,True,False,299,299,L,156.35,68dc3953d4149f0689bd2088089285c4
COVID-1890.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1890.png,COVID/images/COVID-1890.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1890.png,COVID/masks/COVID-1890.png,True,False,299,299,L,157.01,47cb30b163c7f7cbf9d04bee10b42ed3
COVID-1891.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1891.png,COVID/images/COVID-1891.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1891.png,COVID/masks/COVID-1891.png,True,False,299,299,L,149.54,0ac394fffafce3f99723c368274fc854
COVID-1892.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1892.png,COVID/images/COVID-1892.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1892.png,COVID/masks/COVID-1892.png,True,False,299,299,L,156.51,c25db293315c2b8fbc971347d7517aee
COVID-1893.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1893.png,COVID/images/COVID-1893.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1893.png,COVID/masks/COVID-1893.png,True,False,299,299,L,148.58,6545ddd160524aac3c4f33fde083c075
COVID-1894.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1894.png,COVID/images/COVID-1894.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1894.png,COVID/masks/COVID-1894.png,True,False,299,299,L,166.3,0a1bba5afd7cb9a7e8370be5309b81fa
COVID-1895.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1895.png,COVID/images/COVID-1895.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1895.png,COVID/masks/COVID-1895.png,True,False,299,299,L,151.03,2d45ab5d5f63e765efec48d0d2713164
COVID-1896.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1896.png,COVID/images/COVID-1896.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1896.png,COVID/masks/COVID-1896.png,True,False,299,299,L,145.67,4a01a4322a6413c98a12d0d8bdd02c7d
COVID-1897.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1897.png,COVID/images/COVID-1897.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1897.png,COVID/masks/COVID-1897.png,True,False,299,299,L,95.56,a3e6873011b79c6b02d717964a9ea902
COVID-1898.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1898.png,COVID/images/COVID-1898.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1898.png,COVID/masks/COVID-1898.png,True,False,299,299,L,169.47,0c683a17dea47dbef535a2e0dadbd533
COVID-1899.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1899.png,COVID/images/COVID-1899.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1899.png,COVID/masks/COVID-1899.png,True,False,299,299,L,146.81,0a8bdd750dc05b9a15940be26d32b74c
COVID-19.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-19.png,COVID/images/COVID-19.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-19.png,COVID/masks/COVID-19.png,True,False,299,299,L,117.15,89da97b327b4eb7afa8bcdbbe48b5112
COVID-190.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-190.png,COVID/images/COVID-190.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-190.png,COVID/masks/COVID-190.png,True,False,299,299,L,135.76,0693072dfc5ee33da9f07a2c4b32e0d0
COVID-1900.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1900.png,COVID/images/COVID-1900.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1900.png,COVID/masks/COVID-1900.png,True,False,299,299,L,149.53,a3dff31c91181a468d6c87d6dea73190
COVID-1901.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1901.png,COVID/images/COVID-1901.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1901.png,COVID/masks/COVID-1901.png,True,False,299,299,L,158.62,8be284444dbef09e2ab87ce19016134d
COVID-1902.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1902.png,COVID/images/COVID-1902.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1902.png,COVID/masks/COVID-1902.png,True,False,299,299,L,149.96,530e3ed1e6c1d6c8c315ae357927a8ad
COVID-1903.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1903.png,COVID/images/COVID-1903.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1903.png,COVID/masks/COVID-1903.png,True,False,299,299,L,130.32,a49ab1c616237eb27a529015ab159d8f
COVID-1904.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1904.png,COVID/images/COVID-1904.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1904.png,COVID/masks/COVID-1904.png,True,False,299,299,L,158.27,c2c07ca06ee5272d19ba2daac0ae69d2
COVID-1905.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1905.png,COVID/images/COVID-1905.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1905.png,COVID/masks/COVID-1905.png,True,False,299,299,L,155.97,2a0f3e9219773a5596ba2a9d533d4b35
COVID-1906.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1906.png,COVID/images/COVID-1906.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1906.png,COVID/masks/COVID-1906.png,True,False,299,299,L,180.31,7f43ee2eea6b638822d08eb69d21622e
COVID-1907.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1907.png,COVID/images/COVID-1907.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1907.png,COVID/masks/COVID-1907.png,True,False,299,299,L,149.96,268100ce5f3d59edeb1bd9f3964f7987
COVID-1908.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1908.png,COVID/images/COVID-1908.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1908.png,COVID/masks/COVID-1908.png,True,False,299,299,L,165.91,9e7be7c578636bfcba4899f4db4df056
COVID-1909.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1909.png,COVID/images/COVID-1909.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1909.png,COVID/masks/COVID-1909.png,True,False,299,299,L,186.27,38172df30530f1fd352bfd347d3595ae
COVID-191.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-191.png,COVID/images/COVID-191.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-191.png,COVID/masks/COVID-191.png,True,False,299,299,L,173.19,2e3e1ed37922407f8658d8763d6ec069
COVID-1910.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1910.png,COVID/images/COVID-1910.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1910.png,COVID/masks/COVID-1910.png,True,False,299,299,L,164.32,0a5195d6d7e1c5af7c1535dc1e1d0601
COVID-1911.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1911.png,COVID/images/COVID-1911.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1911.png,COVID/masks/COVID-1911.png,True,False,299,299,L,185.11,48f66b3e246aa5556170d0826558f0f5
COVID-1912.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1912.png,COVID/images/COVID-1912.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1912.png,COVID/masks/COVID-1912.png,True,False,299,299,L,161.16,d3ab0dfcc83ebe18aa3aa4b61c037a3c
COVID-1913.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1913.png,COVID/images/COVID-1913.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1913.png,COVID/masks/COVID-1913.png,True,False,299,299,L,120.64,2ce8d4410cb02a3f67c4c9b6823392cf
COVID-1914.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1914.png,COVID/images/COVID-1914.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1914.png,COVID/masks/COVID-1914.png,True,False,299,299,L,69.79,dfa99d2331b4a3bcb7ac2c6066d5eaa0
COVID-1915.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1915.png,COVID/images/COVID-1915.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1915.png,COVID/masks/COVID-1915.png,True,False,299,299,L,158.17,8c9bf5f9153dde9a0002452d9a9cebf5
COVID-1916.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1916.png,COVID/images/COVID-1916.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1916.png,COVID/masks/COVID-1916.png,True,False,299,299,L,163.4,3b5166efb15e849be10e124bcf30c364
COVID-1917.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1917.png,COVID/images/COVID-1917.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1917.png,COVID/masks/COVID-1917.png,True,False,299,299,L,150.62,695d03679faf09d851a3878b8a77333c
COVID-1918.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1918.png,COVID/images/COVID-1918.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1918.png,COVID/masks/COVID-1918.png,True,False,299,299,L,158.16,fb68b0de20024c957d55313ca9aee6dd
COVID-1919.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1919.png,COVID/images/COVID-1919.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1919.png,COVID/masks/COVID-1919.png,True,False,299,299,L,114.34,b27918eb6f36272facfe98e2e0ee6559
COVID-192.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-192.png,COVID/images/COVID-192.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-192.png,COVID/masks/COVID-192.png,True,False,299,299,L,153.24,dd5a041305ecb568c24667e876d2c85a
COVID-1920.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1920.png,COVID/images/COVID-1920.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1920.png,COVID/masks/COVID-1920.png,True,False,299,299,L,146.07,f2907c21e12794f3acd1447a3cc8065e
COVID-1921.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1921.png,COVID/images/COVID-1921.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1921.png,COVID/masks/COVID-1921.png,True,False,299,299,L,136.36,0f9a591731e6c277a2091bd40ec4eeb2
COVID-1922.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1922.png,COVID/images/COVID-1922.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1922.png,COVID/masks/COVID-1922.png,True,False,299,299,L,179.24,5aeccc99275f57d2ae092bd5f61c2215
COVID-1923.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1923.png,COVID/images/COVID-1923.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1923.png,COVID/masks/COVID-1923.png,True,False,299,299,L,77.99,5b29c545527a324a29838ba6f54f70ce
COVID-1924.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1924.png,COVID/images/COVID-1924.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1924.png,COVID/masks/COVID-1924.png,True,False,299,299,L,172.27,ba44ee9f8c0912f6fd823d5c54cf9054
COVID-1925.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1925.png,COVID/images/COVID-1925.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1925.png,COVID/masks/COVID-1925.png,True,False,299,299,L,154.07,efb7fbb0dbd491183ed5bc171a8ece28
COVID-1926.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1926.png,COVID/images/COVID-1926.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1926.png,COVID/masks/COVID-1926.png,True,False,299,299,L,175.43,bba3a7c7da0364b3fa866d4a18fa057f
COVID-1927.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1927.png,COVID/images/COVID-1927.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1927.png,COVID/masks/COVID-1927.png,True,False,299,299,L,160.8,9c64e8ec58b4e11757a7a89815919fac
COVID-1928.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1928.png,COVID/images/COVID-1928.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1928.png,COVID/masks/COVID-1928.png,True,False,299,299,L,134.64,f21f95a24d234c72a9cc2d212627cdae
COVID-1929.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1929.png,COVID/images/COVID-1929.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1929.png,COVID/masks/COVID-1929.png,True,False,299,299,L,141.05,027a493ec5ce229861afc596f59da899
COVID-193.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-193.png,COVID/images/COVID-193.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-193.png,COVID/masks/COVID-193.png,True,False,299,299,L,177.43,eb764a8a67253ea5152eac69af7b739a
COVID-1930.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1930.png,COVID/images/COVID-1930.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1930.png,COVID/masks/COVID-1930.png,True,False,299,299,L,137.38,59bcd945bebfa7e6132d04d1f36a0685
COVID-1931.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1931.png,COVID/images/COVID-1931.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1931.png,COVID/masks/COVID-1931.png,True,False,299,299,L,145.61,648c4ea11c2e55503ded8533bc05180b
COVID-1932.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1932.png,COVID/images/COVID-1932.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1932.png,COVID/masks/COVID-1932.png,True,False,299,299,L,148.72,8253bd2c74b78baf6a457eefa56d9973
COVID-1933.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1933.png,COVID/images/COVID-1933.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1933.png,COVID/masks/COVID-1933.png,True,False,299,299,L,145.34,2ae4b45af5f39a82e59220e7a140c241
COVID-1934.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1934.png,COVID/images/COVID-1934.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1934.png,COVID/masks/COVID-1934.png,True,False,299,299,L,150.86,677d7910572b7dac1edec61934146930
COVID-1935.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1935.png,COVID/images/COVID-1935.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1935.png,COVID/masks/COVID-1935.png,True,False,299,299,L,136.64,16ff2471d15eb1627ea18f0f5bb1c0c9
COVID-1936.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1936.png,COVID/images/COVID-1936.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1936.png,COVID/masks/COVID-1936.png,True,False,299,299,L,100.4,5bd3339aa85846a909eeefebe3674986
COVID-1937.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1937.png,COVID/images/COVID-1937.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1937.png,COVID/masks/COVID-1937.png,True,False,299,299,L,163.22,18b210f7f122af4942f198465fb2e391
COVID-1938.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1938.png,COVID/images/COVID-1938.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1938.png,COVID/masks/COVID-1938.png,True,False,299,299,L,127.77,d6ca5f0e9fca00ff73372aecd3b23627
COVID-1939.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1939.png,COVID/images/COVID-1939.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1939.png,COVID/masks/COVID-1939.png,True,False,299,299,L,174.04,cae19f8899a9703e8fc8f080712de3c0
COVID-194.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-194.png,COVID/images/COVID-194.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-194.png,COVID/masks/COVID-194.png,True,False,299,299,L,125.49,bd75aa058d2c5483ec3f8a159e0d0381
COVID-1940.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1940.png,COVID/images/COVID-1940.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1940.png,COVID/masks/COVID-1940.png,True,False,299,299,L,160.7,131323482c56534c62f627fce2469447
COVID-1941.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1941.png,COVID/images/COVID-1941.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1941.png,COVID/masks/COVID-1941.png,True,False,299,299,L,169.63,64ed5533a7bbd9cab2ef0cc7d8586ecb
COVID-1942.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1942.png,COVID/images/COVID-1942.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1942.png,COVID/masks/COVID-1942.png,True,False,299,299,L,134.49,6c34d73bf97c2533017324459196bc1e
COVID-1943.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1943.png,COVID/images/COVID-1943.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1943.png,COVID/masks/COVID-1943.png,True,False,299,299,L,108.64,d0ac3ac4f79f1885d6039dc812989a68
COVID-1944.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1944.png,COVID/images/COVID-1944.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1944.png,COVID/masks/COVID-1944.png,True,False,299,299,L,178.43,a4485c471c384cb558dc0e6c28f84638
COVID-1945.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1945.png,COVID/images/COVID-1945.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1945.png,COVID/masks/COVID-1945.png,True,False,299,299,L,150.8,ccdeda454115a1b0d9cdccb1e30bf225
COVID-1946.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1946.png,COVID/images/COVID-1946.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1946.png,COVID/masks/COVID-1946.png,True,False,299,299,L,153.33,3410283aeb610af55a852336906cb369
COVID-1947.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1947.png,COVID/images/COVID-1947.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1947.png,COVID/masks/COVID-1947.png,True,False,299,299,L,157.53,4189c15f28114eb6b0deeb8af0e96bc1
COVID-1948.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1948.png,COVID/images/COVID-1948.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1948.png,COVID/masks/COVID-1948.png,True,False,299,299,L,151.36,1c15f019c23bcc450a03714e38299bbd
COVID-1949.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1949.png,COVID/images/COVID-1949.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1949.png,COVID/masks/COVID-1949.png,True,False,299,299,L,123.01,b0fbbd32bb7d51c359d584ef129d7ff5
COVID-195.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-195.png,COVID/images/COVID-195.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-195.png,COVID/masks/COVID-195.png,True,False,299,299,L,190.86,eaebda65a9b3691b3d5e9a8ea1378b1d
COVID-1950.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1950.png,COVID/images/COVID-1950.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1950.png,COVID/masks/COVID-1950.png,True,False,299,299,L,167.87,ada3a2bea6950687fe5f04102e54d2b1
COVID-1951.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1951.png,COVID/images/COVID-1951.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1951.png,COVID/masks/COVID-1951.png,True,False,299,299,L,144.99,886c9babfefed775a68a092856732ee8
COVID-1952.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1952.png,COVID/images/COVID-1952.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1952.png,COVID/masks/COVID-1952.png,True,False,299,299,L,140.2,db64ec94899d760960b6f2dfaa2e8a5a
COVID-1953.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1953.png,COVID/images/COVID-1953.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1953.png,COVID/masks/COVID-1953.png,True,False,299,299,L,95.26,c4a4e643049c098c9e39be38d5392892
COVID-1954.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1954.png,COVID/images/COVID-1954.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1954.png,COVID/masks/COVID-1954.png,True,False,299,299,L,138.45,c6e2eb64899fdfafa6cb1786f2f7954c
COVID-1955.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1955.png,COVID/images/COVID-1955.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1955.png,COVID/masks/COVID-1955.png,True,False,299,299,L,88.41,29eb684bbd6951f96721006b309a80d0
COVID-1956.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1956.png,COVID/images/COVID-1956.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1956.png,COVID/masks/COVID-1956.png,True,False,299,299,L,148.81,dd3b80da0c3d7f72a65d8344a90b2463
COVID-1957.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1957.png,COVID/images/COVID-1957.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1957.png,COVID/masks/COVID-1957.png,True,False,299,299,L,141.44,d8a8da018e808910ba5e81b283313546
COVID-1958.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1958.png,COVID/images/COVID-1958.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1958.png,COVID/masks/COVID-1958.png,True,False,299,299,L,159.67,84108e1c47979b505a1d04e9374454de
COVID-1959.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1959.png,COVID/images/COVID-1959.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1959.png,COVID/masks/COVID-1959.png,True,False,299,299,L,134.53,23150ec02d8940c889fac904fb944d21
COVID-196.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-196.png,COVID/images/COVID-196.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-196.png,COVID/masks/COVID-196.png,True,False,299,299,L,151.52,88e8e31cca70772491bb7e5aa110f531
COVID-1960.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1960.png,COVID/images/COVID-1960.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1960.png,COVID/masks/COVID-1960.png,True,False,299,299,L,93.77,11e470187d0dc0abc27a619128560c27
COVID-1961.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1961.png,COVID/images/COVID-1961.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1961.png,COVID/masks/COVID-1961.png,True,False,299,299,L,115.25,611503e80759219ad55ca52394ef69f9
COVID-1962.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1962.png,COVID/images/COVID-1962.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1962.png,COVID/masks/COVID-1962.png,True,False,299,299,L,120.96,cb8f6cae52dde6dc53405864b545802b
COVID-1963.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1963.png,COVID/images/COVID-1963.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1963.png,COVID/masks/COVID-1963.png,True,False,299,299,L,159.21,99fefa7e683d54c09391d8954de01732
COVID-1964.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1964.png,COVID/images/COVID-1964.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1964.png,COVID/masks/COVID-1964.png,True,False,299,299,L,151.11,0789465e15f13c6d85690c36751210d6
COVID-1965.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1965.png,COVID/images/COVID-1965.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1965.png,COVID/masks/COVID-1965.png,True,False,299,299,L,142.03,cff17e879d475ca731e4929be99411e0
COVID-1966.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1966.png,COVID/images/COVID-1966.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1966.png,COVID/masks/COVID-1966.png,True,False,299,299,L,155.55,349e589afbc18fa7ec3985ae8d504949
COVID-1967.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1967.png,COVID/images/COVID-1967.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1967.png,COVID/masks/COVID-1967.png,True,False,299,299,L,107.21,78184d60399793ceb73f12219f2dfad0
COVID-1968.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1968.png,COVID/images/COVID-1968.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1968.png,COVID/masks/COVID-1968.png,True,False,299,299,L,127.54,39021a770dfcbe9b1170f8b60b781c27
COVID-1969.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1969.png,COVID/images/COVID-1969.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1969.png,COVID/masks/COVID-1969.png,True,False,299,299,L,162.27,504fc23e98b0cc1b9c7bfb92c1a58423
COVID-197.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-197.png,COVID/images/COVID-197.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-197.png,COVID/masks/COVID-197.png,True,False,299,299,L,138.93,4ee8b7749a64a350c13930f7647c6fcb
COVID-1970.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1970.png,COVID/images/COVID-1970.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1970.png,COVID/masks/COVID-1970.png,True,False,299,299,L,140.84,38054c392147e94d56be27bda27ddab9
COVID-1971.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1971.png,COVID/images/COVID-1971.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1971.png,COVID/masks/COVID-1971.png,True,False,299,299,L,163.51,4834f38f3655efffab601abaf0965bb0
COVID-1972.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1972.png,COVID/images/COVID-1972.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1972.png,COVID/masks/COVID-1972.png,True,False,299,299,L,160.29,010890ad6873a8ef6b869136555f14d7
COVID-1973.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1973.png,COVID/images/COVID-1973.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1973.png,COVID/masks/COVID-1973.png,True,False,299,299,L,130.43,ad3a9e764eda4f6ba27db93a009d166e
COVID-1974.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1974.png,COVID/images/COVID-1974.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1974.png,COVID/masks/COVID-1974.png,True,False,299,299,L,158.4,a120f47a472ab2488fda1ac1d9b640a0
COVID-1975.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1975.png,COVID/images/COVID-1975.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1975.png,COVID/masks/COVID-1975.png,True,False,299,299,L,145.39,e6300fca02967544109e19f82c71b0b0
COVID-1976.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1976.png,COVID/images/COVID-1976.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1976.png,COVID/masks/COVID-1976.png,True,False,299,299,L,167.78,6508948994a0a7d8dec68e553c17ca31
COVID-1977.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1977.png,COVID/images/COVID-1977.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1977.png,COVID/masks/COVID-1977.png,True,False,299,299,L,143.01,2ca3839711b47954fe45e948995d80d2
COVID-1978.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1978.png,COVID/images/COVID-1978.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1978.png,COVID/masks/COVID-1978.png,True,False,299,299,L,167.17,ee52de7ebb9d092e259828e4b5165fc9
COVID-1979.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1979.png,COVID/images/COVID-1979.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1979.png,COVID/masks/COVID-1979.png,True,False,299,299,L,163.9,677ba927ec843f3933750db5972ca93c
COVID-198.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-198.png,COVID/images/COVID-198.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-198.png,COVID/masks/COVID-198.png,True,False,299,299,L,142.28,79048507e3e1bc68235699a4e382e494
COVID-1980.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1980.png,COVID/images/COVID-1980.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1980.png,COVID/masks/COVID-1980.png,True,False,299,299,L,148.62,52e3b167474d7337278919e7646f8edf
COVID-1981.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1981.png,COVID/images/COVID-1981.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1981.png,COVID/masks/COVID-1981.png,True,False,299,299,L,142.63,f1baa4feaf415f7ab32d1b0299628ab4
COVID-1982.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1982.png,COVID/images/COVID-1982.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1982.png,COVID/masks/COVID-1982.png,True,False,299,299,L,119.55,5db8ac636829a59ec8f2c40090443e1a
COVID-1983.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1983.png,COVID/images/COVID-1983.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1983.png,COVID/masks/COVID-1983.png,True,False,299,299,L,133.85,ac6086997885a8e20a1391559d4de977
COVID-1984.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1984.png,COVID/images/COVID-1984.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1984.png,COVID/masks/COVID-1984.png,True,False,299,299,L,107.53,0043b28974b7d39b33ec66eb4d11a321
COVID-1985.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1985.png,COVID/images/COVID-1985.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1985.png,COVID/masks/COVID-1985.png,True,False,299,299,L,133.81,9a082efea7bd63f08a022771072dc7d3
COVID-1986.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1986.png,COVID/images/COVID-1986.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1986.png,COVID/masks/COVID-1986.png,True,False,299,299,L,135.84,b6ee18f99b8d96f90be84d5ad1d4c27f
COVID-1987.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1987.png,COVID/images/COVID-1987.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1987.png,COVID/masks/COVID-1987.png,True,False,299,299,L,141.99,9ec06544f754ee0a3654ba62df06b884
COVID-1988.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1988.png,COVID/images/COVID-1988.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1988.png,COVID/masks/COVID-1988.png,True,False,299,299,L,145.85,ec3519cd8e9eb79853173491e6305542
COVID-1989.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1989.png,COVID/images/COVID-1989.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1989.png,COVID/masks/COVID-1989.png,True,False,299,299,L,128.2,438c53360c14d1a78b588ed99688794c
COVID-199.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-199.png,COVID/images/COVID-199.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-199.png,COVID/masks/COVID-199.png,True,False,299,299,L,141.73,24092fbfa0ec8e6bb338d84b319e5eff
COVID-1990.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1990.png,COVID/images/COVID-1990.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1990.png,COVID/masks/COVID-1990.png,True,False,299,299,L,116.6,ad944f37675026dcaf945d7dc70673f4
COVID-1991.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1991.png,COVID/images/COVID-1991.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1991.png,COVID/masks/COVID-1991.png,True,False,299,299,L,133.56,5da2faa5872634720f863298657b5532
COVID-1992.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1992.png,COVID/images/COVID-1992.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1992.png,COVID/masks/COVID-1992.png,True,False,299,299,L,149.25,8e9a0e4ad96ff0fca4494b43fa8a207b
COVID-1993.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1993.png,COVID/images/COVID-1993.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1993.png,COVID/masks/COVID-1993.png,True,False,299,299,L,150.17,86b0711de02981bff8a3eff5d2f555e8
COVID-1994.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1994.png,COVID/images/COVID-1994.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1994.png,COVID/masks/COVID-1994.png,True,False,299,299,L,146.1,49d17bf830353367a4dff2048d64654e
COVID-1995.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1995.png,COVID/images/COVID-1995.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1995.png,COVID/masks/COVID-1995.png,True,False,299,299,L,173.9,a842b3430ac84d58fedab30f9e363971
COVID-1996.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1996.png,COVID/images/COVID-1996.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1996.png,COVID/masks/COVID-1996.png,True,False,299,299,L,171.33,329f9544160992878bb4f0b134253981
COVID-1997.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1997.png,COVID/images/COVID-1997.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1997.png,COVID/masks/COVID-1997.png,True,False,299,299,L,157.28,79d499cdb353260835293cd10477b706
COVID-1998.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1998.png,COVID/images/COVID-1998.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1998.png,COVID/masks/COVID-1998.png,True,False,299,299,L,165.16,9ede06e229116516d0416f76db72dd96
COVID-1999.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-1999.png,COVID/images/COVID-1999.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-1999.png,COVID/masks/COVID-1999.png,True,False,299,299,L,177.26,89d8c74db7840c919556991796f482c4
COVID-2.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2.png,COVID/images/COVID-2.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2.png,COVID/masks/COVID-2.png,True,False,299,299,L,150.64,eba33d9078fb72898901dfc4efb63d8d
COVID-20.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-20.png,COVID/images/COVID-20.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-20.png,COVID/masks/COVID-20.png,True,False,299,299,L,135.49,e52fa582f11088396138cacb52719b49
COVID-200.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-200.png,COVID/images/COVID-200.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-200.png,COVID/masks/COVID-200.png,True,False,299,299,L,149.42,220bf1699ebbb8aeb49111112d2637aa
COVID-2000.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2000.png,COVID/images/COVID-2000.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2000.png,COVID/masks/COVID-2000.png,True,False,299,299,L,175.18,319db74d1d895f55cfd11e89624f616e
COVID-2001.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2001.png,COVID/images/COVID-2001.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2001.png,COVID/masks/COVID-2001.png,True,False,299,299,L,168.35,4d8ca18b3f67bfbde6b2efb73fd8a1d4
COVID-2002.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2002.png,COVID/images/COVID-2002.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2002.png,COVID/masks/COVID-2002.png,True,False,299,299,L,178.82,a39d6001cb3626bfc7d9372e3a1ee295
COVID-2003.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2003.png,COVID/images/COVID-2003.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2003.png,COVID/masks/COVID-2003.png,True,False,299,299,L,176.68,fa5e2d91804f3e760bd648db9d6290bf
COVID-2004.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2004.png,COVID/images/COVID-2004.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2004.png,COVID/masks/COVID-2004.png,True,False,299,299,L,157.41,9658b9548c89826f89fc14e50427405a
COVID-2005.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2005.png,COVID/images/COVID-2005.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2005.png,COVID/masks/COVID-2005.png,True,False,299,299,L,187.51,c16fcdd7ee063ac0485c4f45bfd86e5e
COVID-2006.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2006.png,COVID/images/COVID-2006.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2006.png,COVID/masks/COVID-2006.png,True,False,299,299,L,147.87,c00c6636c8fd669821cca10dbe2c08fe
COVID-2007.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2007.png,COVID/images/COVID-2007.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2007.png,COVID/masks/COVID-2007.png,True,False,299,299,L,159.14,a22bbba1f4efd1a7d0704b9b904d1fab
COVID-2008.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2008.png,COVID/images/COVID-2008.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2008.png,COVID/masks/COVID-2008.png,True,False,299,299,L,159.08,f21078e449e9d416816b882f11b17240
COVID-2009.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2009.png,COVID/images/COVID-2009.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2009.png,COVID/masks/COVID-2009.png,True,False,299,299,L,178.98,b477e9136d1932e4a1943cba4378e443
COVID-201.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-201.png,COVID/images/COVID-201.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-201.png,COVID/masks/COVID-201.png,True,False,299,299,L,97.44,512e15300e6098b1a886d93ef4d61e51
COVID-2010.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2010.png,COVID/images/COVID-2010.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2010.png,COVID/masks/COVID-2010.png,True,False,299,299,L,166.54,54d52b769d1ed5698d159c044c6bd296
COVID-2011.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2011.png,COVID/images/COVID-2011.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2011.png,COVID/masks/COVID-2011.png,True,False,299,299,L,167.61,8bd9697ab90a822bff3c4ef9e3c80928
COVID-2012.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2012.png,COVID/images/COVID-2012.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2012.png,COVID/masks/COVID-2012.png,True,False,299,299,L,146.13,841c6bb72e5ec7f31b3553a428da71db
COVID-2013.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2013.png,COVID/images/COVID-2013.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2013.png,COVID/masks/COVID-2013.png,True,False,299,299,L,166.59,a91340e89bde2d6edbefeda285f9fd71
COVID-2014.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2014.png,COVID/images/COVID-2014.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2014.png,COVID/masks/COVID-2014.png,True,False,299,299,L,144.38,e972c68e610b27da21ecd7282b031077
COVID-2015.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2015.png,COVID/images/COVID-2015.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2015.png,COVID/masks/COVID-2015.png,True,False,299,299,L,79.1,91cb1062542d9fad2def0d67358d1d27
COVID-2016.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2016.png,COVID/images/COVID-2016.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2016.png,COVID/masks/COVID-2016.png,True,False,299,299,L,98.11,fe1aec8950496573cef22842673f3d46
COVID-2017.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2017.png,COVID/images/COVID-2017.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2017.png,COVID/masks/COVID-2017.png,True,False,299,299,L,125.84,60693f64f29eb30937a2cb005d967769
COVID-2018.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2018.png,COVID/images/COVID-2018.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2018.png,COVID/masks/COVID-2018.png,True,False,299,299,L,172.56,7bf4f20f0e35a9329c836f1dddbd2b28
COVID-2019.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2019.png,COVID/images/COVID-2019.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2019.png,COVID/masks/COVID-2019.png,True,False,299,299,L,102.47,24cc3be1d865aa156f8bc3d0334ceee8
COVID-202.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-202.png,COVID/images/COVID-202.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-202.png,COVID/masks/COVID-202.png,True,False,299,299,L,138.81,b11d27b8dcbb42a83b958365531c2fa4
COVID-2020.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2020.png,COVID/images/COVID-2020.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2020.png,COVID/masks/COVID-2020.png,True,False,299,299,L,111.39,1b8274ec8fea210ccb88998cca964c12
COVID-2021.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2021.png,COVID/images/COVID-2021.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2021.png,COVID/masks/COVID-2021.png,True,False,299,299,L,150.06,5386d919e10da47c1f4d9bcb39a5c2f7
COVID-2022.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2022.png,COVID/images/COVID-2022.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2022.png,COVID/masks/COVID-2022.png,True,False,299,299,L,149.22,218e2ec1bd4f16b12b176ec381db56c5
COVID-2023.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2023.png,COVID/images/COVID-2023.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2023.png,COVID/masks/COVID-2023.png,True,False,299,299,L,154.67,2c06a853ddcf7c2593f624b6e9152ec2
COVID-2024.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2024.png,COVID/images/COVID-2024.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2024.png,COVID/masks/COVID-2024.png,True,False,299,299,L,140.57,a819585ba770365953a88d9f6dfaca94
COVID-2025.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2025.png,COVID/images/COVID-2025.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2025.png,COVID/masks/COVID-2025.png,True,False,299,299,L,106.3,f9f4f6b4892b298de0909cfe2d2ea519
COVID-2026.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2026.png,COVID/images/COVID-2026.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2026.png,COVID/masks/COVID-2026.png,True,False,299,299,L,152.64,7c60d5c4a40d55b9de349b22d96d9c6f
COVID-2027.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2027.png,COVID/images/COVID-2027.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2027.png,COVID/masks/COVID-2027.png,True,False,299,299,L,103.28,13c5113b07bf98abdbbb00d5c4e6356a
COVID-2028.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2028.png,COVID/images/COVID-2028.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2028.png,COVID/masks/COVID-2028.png,True,False,299,299,L,104.84,a7bb31ee8dc78ee53cb897eead8b3984
COVID-2029.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2029.png,COVID/images/COVID-2029.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2029.png,COVID/masks/COVID-2029.png,True,False,299,299,L,163.91,5ede53da1b94da087ef0f04fab044d01
COVID-203.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-203.png,COVID/images/COVID-203.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-203.png,COVID/masks/COVID-203.png,True,False,299,299,L,147.14,b9581f28c0be551231ead4beb4ac5e89
COVID-2030.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2030.png,COVID/images/COVID-2030.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2030.png,COVID/masks/COVID-2030.png,True,False,299,299,L,131.97,83533e99ccd1235abcd7150dcadc3f9d
COVID-2031.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2031.png,COVID/images/COVID-2031.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2031.png,COVID/masks/COVID-2031.png,True,False,299,299,L,164.15,e4682d3f97c708d6193ed5706893300e
COVID-2032.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2032.png,COVID/images/COVID-2032.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2032.png,COVID/masks/COVID-2032.png,True,False,299,299,L,166.12,5945eaf4bc4fa8ba05c470070cc19b8e
COVID-2033.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2033.png,COVID/images/COVID-2033.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2033.png,COVID/masks/COVID-2033.png,True,False,299,299,L,169.96,0bce636e9f536b58e27188455f25345b
COVID-2034.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2034.png,COVID/images/COVID-2034.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2034.png,COVID/masks/COVID-2034.png,True,False,299,299,L,144.85,c1a0fcb2bd4ac3dbcaae468aa72ee493
COVID-2035.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2035.png,COVID/images/COVID-2035.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2035.png,COVID/masks/COVID-2035.png,True,False,299,299,L,142.67,e8994cb65824c127a6e68b63e7295328
COVID-2036.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2036.png,COVID/images/COVID-2036.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2036.png,COVID/masks/COVID-2036.png,True,False,299,299,L,145.75,2e1934b5d9e7774eb1924f3c32cdcc93
COVID-2037.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2037.png,COVID/images/COVID-2037.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2037.png,COVID/masks/COVID-2037.png,True,False,299,299,L,139.99,5fb847e56e7b6353798a5a6036af84a5
COVID-2038.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2038.png,COVID/images/COVID-2038.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2038.png,COVID/masks/COVID-2038.png,True,False,299,299,L,144.89,6b35e4013751810e8b6670ad34448de2
COVID-2039.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2039.png,COVID/images/COVID-2039.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2039.png,COVID/masks/COVID-2039.png,True,False,299,299,L,147.52,5f8c31fbace5832cbd7001d7c3aba95c
COVID-204.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-204.png,COVID/images/COVID-204.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-204.png,COVID/masks/COVID-204.png,True,False,299,299,L,164.64,595ab3e234a3f6a0ec04d3dd3f44b244
COVID-2040.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2040.png,COVID/images/COVID-2040.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2040.png,COVID/masks/COVID-2040.png,True,False,299,299,L,160.01,dba1564f773efd0a0fe1805016d8e8f0
COVID-2041.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2041.png,COVID/images/COVID-2041.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2041.png,COVID/masks/COVID-2041.png,True,False,299,299,L,158.35,f5311029048d7763295d2a1cbe4d7a6f
COVID-2042.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2042.png,COVID/images/COVID-2042.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2042.png,COVID/masks/COVID-2042.png,True,False,299,299,L,190.73,67598fe43b1fe61243cdf39692cabdf1
COVID-2043.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2043.png,COVID/images/COVID-2043.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2043.png,COVID/masks/COVID-2043.png,True,False,299,299,L,134.1,067802e523059419783118e2566e41c1
COVID-2044.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2044.png,COVID/images/COVID-2044.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2044.png,COVID/masks/COVID-2044.png,True,False,299,299,L,130.87,b0a5a6960751a6102df5f6d25d84fd0b
COVID-2045.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2045.png,COVID/images/COVID-2045.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2045.png,COVID/masks/COVID-2045.png,True,False,299,299,L,138.6,841ae1074827c74a8ef61811ee322290
COVID-2046.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2046.png,COVID/images/COVID-2046.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2046.png,COVID/masks/COVID-2046.png,True,False,299,299,L,135.68,981002d814e2d28c5782eba5e11b12a9
COVID-2047.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2047.png,COVID/images/COVID-2047.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2047.png,COVID/masks/COVID-2047.png,True,False,299,299,L,167.09,7d35c8c2d4cdf60adaf640439fe4a3c0
COVID-2048.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2048.png,COVID/images/COVID-2048.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2048.png,COVID/masks/COVID-2048.png,True,False,299,299,L,147.0,6bf4b43f9267542d564a433c93f02121
COVID-2049.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2049.png,COVID/images/COVID-2049.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2049.png,COVID/masks/COVID-2049.png,True,False,299,299,L,100.57,92d615a8145df85b93e52fea98dfd95b
COVID-205.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-205.png,COVID/images/COVID-205.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-205.png,COVID/masks/COVID-205.png,True,False,299,299,L,153.08,763c3b1406e74d98ead3915446660355
COVID-2050.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2050.png,COVID/images/COVID-2050.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2050.png,COVID/masks/COVID-2050.png,True,False,299,299,L,152.36,457685a64934481146631f7528779a3c
COVID-2051.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2051.png,COVID/images/COVID-2051.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2051.png,COVID/masks/COVID-2051.png,True,False,299,299,L,152.34,57f09f77e438a962cc68b766ea961d76
COVID-2052.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2052.png,COVID/images/COVID-2052.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2052.png,COVID/masks/COVID-2052.png,True,False,299,299,L,158.35,593ca0058e45409c7ac5e978a3fceb9a
COVID-2053.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2053.png,COVID/images/COVID-2053.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2053.png,COVID/masks/COVID-2053.png,True,False,299,299,L,165.16,a95aefbbc194ded6ecb5d9dfa4c51b61
COVID-2054.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2054.png,COVID/images/COVID-2054.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2054.png,COVID/masks/COVID-2054.png,True,False,299,299,L,164.13,12588d2c02d77b3149df84e404ab467d
COVID-2055.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2055.png,COVID/images/COVID-2055.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2055.png,COVID/masks/COVID-2055.png,True,False,299,299,L,157.88,15b5022272c378b323ba9ccb843dd90a
COVID-2056.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2056.png,COVID/images/COVID-2056.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2056.png,COVID/masks/COVID-2056.png,True,False,299,299,L,157.89,8a582bfbc6dfd039cbb0cf2aea1aaf35
COVID-2057.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2057.png,COVID/images/COVID-2057.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2057.png,COVID/masks/COVID-2057.png,True,False,299,299,L,138.53,a53bd9007428c6cca4f87fe4ea8494ab
COVID-2058.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2058.png,COVID/images/COVID-2058.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2058.png,COVID/masks/COVID-2058.png,True,False,299,299,L,164.36,510eb64c9b269ce2ea5b25493ca071dc
COVID-2059.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2059.png,COVID/images/COVID-2059.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2059.png,COVID/masks/COVID-2059.png,True,False,299,299,L,175.2,3ef359fe9dd45be20b2c131536c9e9fc
COVID-206.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-206.png,COVID/images/COVID-206.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-206.png,COVID/masks/COVID-206.png,True,False,299,299,L,163.01,701686ce58c0243b00d620f374ae52b5
COVID-2060.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2060.png,COVID/images/COVID-2060.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2060.png,COVID/masks/COVID-2060.png,True,False,299,299,L,187.55,8cac0ee69a548ea938b831cb3b9dad25
COVID-2061.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2061.png,COVID/images/COVID-2061.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2061.png,COVID/masks/COVID-2061.png,True,False,299,299,L,152.41,0eb7329e5665eae4f32f8f69ebcaaf36
COVID-2062.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2062.png,COVID/images/COVID-2062.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2062.png,COVID/masks/COVID-2062.png,True,False,299,299,L,172.46,7c67e38d8e1f621f83aebf54ef0bb2f4
COVID-2063.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2063.png,COVID/images/COVID-2063.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2063.png,COVID/masks/COVID-2063.png,True,False,299,299,L,154.28,ab0b6d07228be500f8a9b134affd586f
COVID-2064.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2064.png,COVID/images/COVID-2064.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2064.png,COVID/masks/COVID-2064.png,True,False,299,299,L,179.06,da7c11d32cd11a23bcf1a45797e32eb3
COVID-2065.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2065.png,COVID/images/COVID-2065.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2065.png,COVID/masks/COVID-2065.png,True,False,299,299,L,148.37,8ca79e0470ac4bb56036538b5649823a
COVID-2066.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2066.png,COVID/images/COVID-2066.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2066.png,COVID/masks/COVID-2066.png,True,False,299,299,L,138.73,fb59d081c13e949a34cbb5bed42b1851
COVID-2067.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2067.png,COVID/images/COVID-2067.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2067.png,COVID/masks/COVID-2067.png,True,False,299,299,L,141.49,d6a57ec518b0c17f8db5e7294a39393d
COVID-2068.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2068.png,COVID/images/COVID-2068.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2068.png,COVID/masks/COVID-2068.png,True,False,299,299,L,154.55,039f569768715a6a7cfe7d0be3c9e8ed
COVID-2069.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2069.png,COVID/images/COVID-2069.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2069.png,COVID/masks/COVID-2069.png,True,False,299,299,L,153.22,512b5f3ebd066df12930c06accab6721
COVID-207.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-207.png,COVID/images/COVID-207.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-207.png,COVID/masks/COVID-207.png,True,False,299,299,L,98.82,23f907f98016e861c289226dc28e6684
COVID-2070.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2070.png,COVID/images/COVID-2070.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2070.png,COVID/masks/COVID-2070.png,True,False,299,299,L,150.33,e89d0c348da8d399f26f7e1ae2b0d2f2
COVID-2071.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2071.png,COVID/images/COVID-2071.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2071.png,COVID/masks/COVID-2071.png,True,False,299,299,L,123.87,ced71a9a8a3a0850851c9ab386f6cd61
COVID-2072.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2072.png,COVID/images/COVID-2072.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2072.png,COVID/masks/COVID-2072.png,True,False,299,299,L,182.09,d85104034429b4766304b403a8f5c947
COVID-2073.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2073.png,COVID/images/COVID-2073.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2073.png,COVID/masks/COVID-2073.png,True,False,299,299,L,160.06,50137721d3090c132baed85beaa7be23
COVID-2074.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2074.png,COVID/images/COVID-2074.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2074.png,COVID/masks/COVID-2074.png,True,False,299,299,L,144.11,4e2dc65b9d125e32c8c13990fb0a5051
COVID-2075.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2075.png,COVID/images/COVID-2075.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2075.png,COVID/masks/COVID-2075.png,True,False,299,299,L,128.89,8669aa862f5db31cdc6c7b8afc7589dc
COVID-2076.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2076.png,COVID/images/COVID-2076.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2076.png,COVID/masks/COVID-2076.png,True,False,299,299,L,151.93,9f2809e084134b52aa73b4a3e5d579eb
COVID-2077.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2077.png,COVID/images/COVID-2077.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2077.png,COVID/masks/COVID-2077.png,True,False,299,299,L,166.99,2aa28a052e51c5c95943c2072c4b3f32
COVID-2078.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2078.png,COVID/images/COVID-2078.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2078.png,COVID/masks/COVID-2078.png,True,False,299,299,L,137.61,69bf625aba66de8bd444234a9a84d43c
COVID-2079.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2079.png,COVID/images/COVID-2079.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2079.png,COVID/masks/COVID-2079.png,True,False,299,299,L,159.18,f3f64ab8e37f521fa1c3d8c113c618a6
COVID-208.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-208.png,COVID/images/COVID-208.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-208.png,COVID/masks/COVID-208.png,True,False,299,299,L,147.21,244a500d246faca43bc1f9032deba253
COVID-2080.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2080.png,COVID/images/COVID-2080.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2080.png,COVID/masks/COVID-2080.png,True,False,299,299,L,167.91,f5cb2f1095c5e2956c7a3adcd8bbeaf6
COVID-2081.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2081.png,COVID/images/COVID-2081.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2081.png,COVID/masks/COVID-2081.png,True,False,299,299,L,132.88,516482f1c59e87e0ff4190c9dbfb4f2e
COVID-2082.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2082.png,COVID/images/COVID-2082.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2082.png,COVID/masks/COVID-2082.png,True,False,299,299,L,179.0,7583bff7a8710c663a96cbb14bfaf326
COVID-2083.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2083.png,COVID/images/COVID-2083.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2083.png,COVID/masks/COVID-2083.png,True,False,299,299,L,161.8,38a5ecf3ae9fb0b96c020d0717f61ed6
COVID-2084.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2084.png,COVID/images/COVID-2084.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2084.png,COVID/masks/COVID-2084.png,True,False,299,299,L,130.31,0cdd53b939b058525c75486a7333fdcb
COVID-2085.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2085.png,COVID/images/COVID-2085.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2085.png,COVID/masks/COVID-2085.png,True,False,299,299,L,130.31,0cdd53b939b058525c75486a7333fdcb
COVID-2086.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2086.png,COVID/images/COVID-2086.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2086.png,COVID/masks/COVID-2086.png,True,False,299,299,L,124.91,4371b05f4328e589fae18575dddd7583
COVID-2087.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2087.png,COVID/images/COVID-2087.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2087.png,COVID/masks/COVID-2087.png,True,False,299,299,L,134.66,1b3bd1cc4289186ceebad7aca4a1f0e2
COVID-2088.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2088.png,COVID/images/COVID-2088.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2088.png,COVID/masks/COVID-2088.png,True,False,299,299,L,118.2,407e8d8039847b405d14cb1eac35cf07
COVID-2089.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2089.png,COVID/images/COVID-2089.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2089.png,COVID/masks/COVID-2089.png,True,False,299,299,L,149.78,baaf853a08e34d7a4a769f33981ae877
COVID-209.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-209.png,COVID/images/COVID-209.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-209.png,COVID/masks/COVID-209.png,True,False,299,299,L,126.04,5628f78926b2f6fc5ec0bb03218ea73e
COVID-2090.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2090.png,COVID/images/COVID-2090.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2090.png,COVID/masks/COVID-2090.png,True,False,299,299,L,119.34,6193a0bd4ae2edf2889cf58a0c7c7b0d
COVID-2091.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2091.png,COVID/images/COVID-2091.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2091.png,COVID/masks/COVID-2091.png,True,False,299,299,L,155.65,27784ea3190e1ca5d13fc3fd7a6117b3
COVID-2092.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2092.png,COVID/images/COVID-2092.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2092.png,COVID/masks/COVID-2092.png,True,False,299,299,L,125.86,da8f4ddecd7ed7d26c9132fd6fdfb463
COVID-2093.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2093.png,COVID/images/COVID-2093.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2093.png,COVID/masks/COVID-2093.png,True,False,299,299,L,127.99,efc67e8a0f050bd3dd1060f1ea4d597c
COVID-2094.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2094.png,COVID/images/COVID-2094.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2094.png,COVID/masks/COVID-2094.png,True,False,299,299,L,176.47,70069d71ad346c042c5936a278bd3ae6
COVID-2095.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2095.png,COVID/images/COVID-2095.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2095.png,COVID/masks/COVID-2095.png,True,False,299,299,L,155.65,43b2685345ac16fede2bfd08daf4753d
COVID-2096.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2096.png,COVID/images/COVID-2096.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2096.png,COVID/masks/COVID-2096.png,True,False,299,299,L,150.78,180b80ab61a98843a96a782897449879
COVID-2097.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2097.png,COVID/images/COVID-2097.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2097.png,COVID/masks/COVID-2097.png,True,False,299,299,L,167.69,258b3446796d592eb7e3da4413d452d7
COVID-2098.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2098.png,COVID/images/COVID-2098.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2098.png,COVID/masks/COVID-2098.png,True,False,299,299,L,166.73,52f1694724a978bbc97b8d766c821d36
COVID-2099.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2099.png,COVID/images/COVID-2099.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2099.png,COVID/masks/COVID-2099.png,True,False,299,299,L,133.54,e61a89ac24b89658e7b52633d1d94858
COVID-21.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-21.png,COVID/images/COVID-21.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-21.png,COVID/masks/COVID-21.png,True,False,299,299,L,151.38,a00f1e48eaebd032eee090def7795dac
COVID-210.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-210.png,COVID/images/COVID-210.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-210.png,COVID/masks/COVID-210.png,True,False,299,299,L,153.48,55ab0cbc21b40a685afd928495c72baf
COVID-2100.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2100.png,COVID/images/COVID-2100.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2100.png,COVID/masks/COVID-2100.png,True,False,299,299,L,159.57,448fbacf530fb24a671e27edab3de28d
COVID-2101.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2101.png,COVID/images/COVID-2101.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2101.png,COVID/masks/COVID-2101.png,True,False,299,299,L,154.45,d2f24aaa93af59662d78654344760ef3
COVID-2102.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2102.png,COVID/images/COVID-2102.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2102.png,COVID/masks/COVID-2102.png,True,False,299,299,L,152.88,3d3dc6955f183d6e1bf3e58f9f42c881
COVID-2103.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2103.png,COVID/images/COVID-2103.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2103.png,COVID/masks/COVID-2103.png,True,False,299,299,L,167.89,ec76657a0b5a3e0b9a2e08ff7f551d02
COVID-2104.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2104.png,COVID/images/COVID-2104.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2104.png,COVID/masks/COVID-2104.png,True,False,299,299,L,157.54,9878bdad09c1128633da3d5f3024f715
COVID-2105.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2105.png,COVID/images/COVID-2105.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2105.png,COVID/masks/COVID-2105.png,True,False,299,299,L,68.16,8894ea6e0121dbd7318861c130670166
COVID-2106.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2106.png,COVID/images/COVID-2106.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2106.png,COVID/masks/COVID-2106.png,True,False,299,299,L,180.04,6633c9f335bc7ac2a4ae7f05980efc0c
COVID-2107.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2107.png,COVID/images/COVID-2107.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2107.png,COVID/masks/COVID-2107.png,True,False,299,299,L,56.21,3fe312124dd6506ad892d92028aaf008
COVID-2108.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2108.png,COVID/images/COVID-2108.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2108.png,COVID/masks/COVID-2108.png,True,False,299,299,L,129.09,87b2f39500f69b1c074adf238ac4a7d6
COVID-2109.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2109.png,COVID/images/COVID-2109.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2109.png,COVID/masks/COVID-2109.png,True,False,299,299,L,167.11,028adffb0ddfdb90f1a830782089eb19
COVID-211.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-211.png,COVID/images/COVID-211.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-211.png,COVID/masks/COVID-211.png,True,False,299,299,L,142.35,f6526535895648b81a3fda1fef6f0470
COVID-2110.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2110.png,COVID/images/COVID-2110.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2110.png,COVID/masks/COVID-2110.png,True,False,299,299,L,173.95,ddff955b2a164be7e16419d3b7bd4d2c
COVID-2111.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2111.png,COVID/images/COVID-2111.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2111.png,COVID/masks/COVID-2111.png,True,False,299,299,L,216.34,2d5a41c78bdc1c866c8408b27d682ce7
COVID-2112.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2112.png,COVID/images/COVID-2112.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2112.png,COVID/masks/COVID-2112.png,True,False,299,299,L,137.9,647431ce468590b2e45e4c78948b3171
COVID-2113.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2113.png,COVID/images/COVID-2113.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2113.png,COVID/masks/COVID-2113.png,True,False,299,299,L,159.35,45c7fa44c2e8caff81cf49df3ec54c29
COVID-2114.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2114.png,COVID/images/COVID-2114.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2114.png,COVID/masks/COVID-2114.png,True,False,299,299,L,152.27,0e2f264b9b5e285d48aabfb70cb1d2bd
COVID-2115.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2115.png,COVID/images/COVID-2115.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2115.png,COVID/masks/COVID-2115.png,True,False,299,299,L,113.83,92bb4e810ecdb840d471dd9c136159ae
COVID-2116.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2116.png,COVID/images/COVID-2116.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2116.png,COVID/masks/COVID-2116.png,True,False,299,299,L,180.29,ed4afcbd2755d28a3fa11603857cd695
COVID-2117.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2117.png,COVID/images/COVID-2117.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2117.png,COVID/masks/COVID-2117.png,True,False,299,299,L,139.65,2393f374f1a148fa1d3a0ef7a7affbcb
COVID-2118.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2118.png,COVID/images/COVID-2118.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2118.png,COVID/masks/COVID-2118.png,True,False,299,299,L,145.66,69e18dfe2c88fb9aababcfe216897872
COVID-2119.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2119.png,COVID/images/COVID-2119.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2119.png,COVID/masks/COVID-2119.png,True,False,299,299,L,140.94,60bef4855fcb8a65db5b8d4daddaf3a6
COVID-212.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-212.png,COVID/images/COVID-212.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-212.png,COVID/masks/COVID-212.png,True,False,299,299,L,154.89,dc8c004df41d398606ad26b18ffd3284
COVID-2120.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2120.png,COVID/images/COVID-2120.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2120.png,COVID/masks/COVID-2120.png,True,False,299,299,L,142.22,75a5d68e71142d58eba576fdd91fda96
COVID-2121.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2121.png,COVID/images/COVID-2121.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2121.png,COVID/masks/COVID-2121.png,True,False,299,299,L,173.45,66afa56b980b45fcf42a56928e872ef0
COVID-2122.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2122.png,COVID/images/COVID-2122.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2122.png,COVID/masks/COVID-2122.png,True,False,299,299,L,116.2,60f0d1b570e97cb38c66510f57271d5d
COVID-2123.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2123.png,COVID/images/COVID-2123.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2123.png,COVID/masks/COVID-2123.png,True,False,299,299,L,161.05,fb9f45be08d474aa23f30e96f35da07a
COVID-2124.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2124.png,COVID/images/COVID-2124.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2124.png,COVID/masks/COVID-2124.png,True,False,299,299,L,140.92,d7cf0b3d285dea4ed0e5fdf7d7b69eda
COVID-2125.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2125.png,COVID/images/COVID-2125.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2125.png,COVID/masks/COVID-2125.png,True,False,299,299,L,109.36,0dd5d7d47747ffdd87130da373ff0884
COVID-2126.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2126.png,COVID/images/COVID-2126.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2126.png,COVID/masks/COVID-2126.png,True,False,299,299,L,134.77,177a5127f0e72481e9abe708313f3aea
COVID-2127.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2127.png,COVID/images/COVID-2127.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2127.png,COVID/masks/COVID-2127.png,True,False,299,299,L,128.19,40e7dd7fe4639db92254f5c54b7908a9
COVID-2128.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2128.png,COVID/images/COVID-2128.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2128.png,COVID/masks/COVID-2128.png,True,False,299,299,L,144.94,af9a89551cd0a030927658678540f82b
COVID-2129.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2129.png,COVID/images/COVID-2129.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2129.png,COVID/masks/COVID-2129.png,True,False,299,299,L,119.56,e22761ef3e5377e5d1c7c90e9416b52d
COVID-213.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-213.png,COVID/images/COVID-213.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-213.png,COVID/masks/COVID-213.png,True,False,299,299,L,132.68,94df5163e9f970221c463e7f51eb3315
COVID-2130.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2130.png,COVID/images/COVID-2130.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2130.png,COVID/masks/COVID-2130.png,True,False,299,299,L,156.91,d02ad42301c6186b2f787aae02031bff
COVID-2131.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2131.png,COVID/images/COVID-2131.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2131.png,COVID/masks/COVID-2131.png,True,False,299,299,L,118.01,b118b2e0ea7e2714b3ecd999d906055d
COVID-2132.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2132.png,COVID/images/COVID-2132.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2132.png,COVID/masks/COVID-2132.png,True,False,299,299,L,143.9,2c15aef8934fa4e4472f37297faf9b46
COVID-2133.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2133.png,COVID/images/COVID-2133.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2133.png,COVID/masks/COVID-2133.png,True,False,299,299,L,148.85,51a85ea6952854a90b5866fd7376a6c0
COVID-2134.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2134.png,COVID/images/COVID-2134.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2134.png,COVID/masks/COVID-2134.png,True,False,299,299,L,180.01,86fb5317cee3be65cfde1c6e7cda42ec
COVID-2135.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2135.png,COVID/images/COVID-2135.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2135.png,COVID/masks/COVID-2135.png,True,False,299,299,L,130.92,32a007de1a6c5e17f8dc945274649a92
COVID-2136.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2136.png,COVID/images/COVID-2136.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2136.png,COVID/masks/COVID-2136.png,True,False,299,299,L,144.36,3a3922b4d62c38d64458f6ec2898ddd3
COVID-2137.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2137.png,COVID/images/COVID-2137.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2137.png,COVID/masks/COVID-2137.png,True,False,299,299,L,79.14,cfea0d26a63404f2c1f67761ed545c75
COVID-2138.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2138.png,COVID/images/COVID-2138.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2138.png,COVID/masks/COVID-2138.png,True,False,299,299,L,142.94,d3fb0d944896f82f66104d068363cabb
COVID-2139.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2139.png,COVID/images/COVID-2139.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2139.png,COVID/masks/COVID-2139.png,True,False,299,299,L,177.7,1a56c95f0b5043ba33929f009d0564cc
COVID-214.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-214.png,COVID/images/COVID-214.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-214.png,COVID/masks/COVID-214.png,True,False,299,299,L,121.73,fb680e9b139079108276784ee21e20ec
COVID-2140.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2140.png,COVID/images/COVID-2140.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2140.png,COVID/masks/COVID-2140.png,True,False,299,299,L,160.72,f4dc9d9f0637f3bde88b4c4d38782029
COVID-2141.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2141.png,COVID/images/COVID-2141.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2141.png,COVID/masks/COVID-2141.png,True,False,299,299,L,170.47,dd1a1fdf8729734b317bf5a249ca56c3
COVID-2142.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2142.png,COVID/images/COVID-2142.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2142.png,COVID/masks/COVID-2142.png,True,False,299,299,L,161.36,470f3dc3c98de60b4483058e475f84cf
COVID-2143.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2143.png,COVID/images/COVID-2143.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2143.png,COVID/masks/COVID-2143.png,True,False,299,299,L,146.93,1bc607b239ca0d0ec1d19530b78ae748
COVID-2144.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2144.png,COVID/images/COVID-2144.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2144.png,COVID/masks/COVID-2144.png,True,False,299,299,L,109.37,cc54e4cd23a752b2a43a02dabb7e5b56
COVID-2145.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2145.png,COVID/images/COVID-2145.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2145.png,COVID/masks/COVID-2145.png,True,False,299,299,L,147.06,9ae02a6b36d36e4349decd2f87173a3c
COVID-2146.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2146.png,COVID/images/COVID-2146.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2146.png,COVID/masks/COVID-2146.png,True,False,299,299,L,146.2,7ae5520b0a67fd81dd6ea5a0317e1e9d
COVID-2147.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2147.png,COVID/images/COVID-2147.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2147.png,COVID/masks/COVID-2147.png,True,False,299,299,L,89.29,9f78fed4732f6bf95bd0519df4ea2de9
COVID-2148.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2148.png,COVID/images/COVID-2148.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2148.png,COVID/masks/COVID-2148.png,True,False,299,299,L,114.05,ae56dae0c786a8120e8b17a7ed6e6d91
COVID-2149.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2149.png,COVID/images/COVID-2149.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2149.png,COVID/masks/COVID-2149.png,True,False,299,299,L,151.51,d4c82523cd9d929886848ec33da5b741
COVID-215.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-215.png,COVID/images/COVID-215.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-215.png,COVID/masks/COVID-215.png,True,False,299,299,L,152.64,2217adc0ebe170488a179b479ce84a61
COVID-2150.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2150.png,COVID/images/COVID-2150.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2150.png,COVID/masks/COVID-2150.png,True,False,299,299,L,172.29,dd20be629302627cac5b9e2aff949912
COVID-2151.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2151.png,COVID/images/COVID-2151.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2151.png,COVID/masks/COVID-2151.png,True,False,299,299,L,143.78,10aedbbca0faefd025297a2960794d1d
COVID-2152.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2152.png,COVID/images/COVID-2152.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2152.png,COVID/masks/COVID-2152.png,True,False,299,299,L,133.33,ff7b83b6141c5a2186013513710e15ae
COVID-2153.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2153.png,COVID/images/COVID-2153.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2153.png,COVID/masks/COVID-2153.png,True,False,299,299,L,69.9,badefae4c8670742b96f55babca955cb
COVID-2154.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2154.png,COVID/images/COVID-2154.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2154.png,COVID/masks/COVID-2154.png,True,False,299,299,L,137.89,5fd427e3556703984fe333cc2cab0a28
COVID-2155.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2155.png,COVID/images/COVID-2155.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2155.png,COVID/masks/COVID-2155.png,True,False,299,299,L,134.94,2de6f5bff3de96739bed84c7e20d1703
COVID-2156.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2156.png,COVID/images/COVID-2156.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2156.png,COVID/masks/COVID-2156.png,True,False,299,299,L,154.18,2c4a51dafc6c50456b48614b4aea2604
COVID-2157.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2157.png,COVID/images/COVID-2157.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2157.png,COVID/masks/COVID-2157.png,True,False,299,299,L,158.21,730e714392564909493c91cbf2f8101f
COVID-2158.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2158.png,COVID/images/COVID-2158.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2158.png,COVID/masks/COVID-2158.png,True,False,299,299,L,153.56,1c9c36b9b57b9dbbfbecc28cdba0ed34
COVID-2159.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2159.png,COVID/images/COVID-2159.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2159.png,COVID/masks/COVID-2159.png,True,False,299,299,L,130.35,88750bd4bb930b2b28073377a9653cd1
COVID-216.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-216.png,COVID/images/COVID-216.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-216.png,COVID/masks/COVID-216.png,True,False,299,299,L,188.11,ea8ac7cae619ff0cdf9893e1ad1c41f0
COVID-2160.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2160.png,COVID/images/COVID-2160.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2160.png,COVID/masks/COVID-2160.png,True,False,299,299,L,79.75,d93a288e2d9fc8d08d1939329620aa46
COVID-2161.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2161.png,COVID/images/COVID-2161.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2161.png,COVID/masks/COVID-2161.png,True,False,299,299,L,136.69,fa91d7da8b0632d948a5b24b3967f0f8
COVID-2162.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2162.png,COVID/images/COVID-2162.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2162.png,COVID/masks/COVID-2162.png,True,False,299,299,L,181.26,056d69c6e0ad1e69891f4e828278604b
COVID-2163.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2163.png,COVID/images/COVID-2163.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2163.png,COVID/masks/COVID-2163.png,True,False,299,299,L,133.78,b1dd370a6ed6f0a816846d4fadb44776
COVID-2164.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2164.png,COVID/images/COVID-2164.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2164.png,COVID/masks/COVID-2164.png,True,False,299,299,L,154.72,998b3b10c49f857fe97a1f811f9be087
COVID-2165.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2165.png,COVID/images/COVID-2165.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2165.png,COVID/masks/COVID-2165.png,True,False,299,299,L,142.87,a1ee49a3a8c79045ddf8fa63a937d9b8
COVID-2166.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2166.png,COVID/images/COVID-2166.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2166.png,COVID/masks/COVID-2166.png,True,False,299,299,L,154.46,24d947351c14867a3c0239c4dbf4ada9
COVID-2167.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2167.png,COVID/images/COVID-2167.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2167.png,COVID/masks/COVID-2167.png,True,False,299,299,L,167.85,850ed2c2417cb1aa31c88613fc6dd166
COVID-2168.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2168.png,COVID/images/COVID-2168.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2168.png,COVID/masks/COVID-2168.png,True,False,299,299,L,144.27,95bf51c5c1dc22a7f81ef1e62593d734
COVID-2169.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2169.png,COVID/images/COVID-2169.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2169.png,COVID/masks/COVID-2169.png,True,False,299,299,L,135.02,973c61134051ff2f949c9a1fcfb268cc
COVID-217.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-217.png,COVID/images/COVID-217.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-217.png,COVID/masks/COVID-217.png,True,False,299,299,L,167.24,e65ffbb682208dfd8491eb68cdc4ed97
COVID-2170.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2170.png,COVID/images/COVID-2170.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2170.png,COVID/masks/COVID-2170.png,True,False,299,299,L,120.56,8d24b03ef39d8b60aaa88cfd40c033a3
COVID-2171.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2171.png,COVID/images/COVID-2171.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2171.png,COVID/masks/COVID-2171.png,True,False,299,299,L,175.57,6818d9f5fac79c074f5f1398fc618541
COVID-2172.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2172.png,COVID/images/COVID-2172.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2172.png,COVID/masks/COVID-2172.png,True,False,299,299,L,159.85,65db122babe88e10188d09344606594c
COVID-2173.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2173.png,COVID/images/COVID-2173.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2173.png,COVID/masks/COVID-2173.png,True,False,299,299,L,168.93,cda3e146f6c4b8aed2431c16af5b3315
COVID-2174.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2174.png,COVID/images/COVID-2174.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2174.png,COVID/masks/COVID-2174.png,True,False,299,299,L,118.9,2cf73e894d062ab75a6e3ec2dfd29a49
COVID-2175.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2175.png,COVID/images/COVID-2175.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2175.png,COVID/masks/COVID-2175.png,True,False,299,299,L,171.32,ae7996f62d10fc50f43c446a89af65e5
COVID-2176.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2176.png,COVID/images/COVID-2176.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2176.png,COVID/masks/COVID-2176.png,True,False,299,299,L,154.9,faff2e70a7243de8ffa3f906f409c41c
COVID-2177.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2177.png,COVID/images/COVID-2177.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2177.png,COVID/masks/COVID-2177.png,True,False,299,299,L,145.92,f138f44335445dc55c3416c47fafff9d
COVID-2178.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2178.png,COVID/images/COVID-2178.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2178.png,COVID/masks/COVID-2178.png,True,False,299,299,L,150.64,ee9fd93a66b0a4f67f2175f35ebe164e
COVID-2179.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2179.png,COVID/images/COVID-2179.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2179.png,COVID/masks/COVID-2179.png,True,False,299,299,L,111.61,1739409ee08e2bc9df73e6e3de49a738
COVID-218.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-218.png,COVID/images/COVID-218.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-218.png,COVID/masks/COVID-218.png,True,False,299,299,L,107.4,b648162a043140c8095e8ebd8a176d8d
COVID-2180.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2180.png,COVID/images/COVID-2180.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2180.png,COVID/masks/COVID-2180.png,True,False,299,299,L,149.41,6efe5a57aa1118c9dc46f9fa1eef0dd6
COVID-2181.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2181.png,COVID/images/COVID-2181.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2181.png,COVID/masks/COVID-2181.png,True,False,299,299,L,165.95,2996629915fbcf2d824d8e95d03611ec
COVID-2182.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2182.png,COVID/images/COVID-2182.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2182.png,COVID/masks/COVID-2182.png,True,False,299,299,L,59.91,2f4a0f838b3a98c2723cd72f6782244c
COVID-2183.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2183.png,COVID/images/COVID-2183.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2183.png,COVID/masks/COVID-2183.png,True,False,299,299,L,144.8,b92fa40e36bed2025f2dbfe6a6391052
COVID-2184.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2184.png,COVID/images/COVID-2184.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2184.png,COVID/masks/COVID-2184.png,True,False,299,299,L,121.15,022737055892381b0a8411f7969fbaff
COVID-2185.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2185.png,COVID/images/COVID-2185.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2185.png,COVID/masks/COVID-2185.png,True,False,299,299,L,147.38,919817723ab8acb84a0748382f7fa0da
COVID-2186.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2186.png,COVID/images/COVID-2186.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2186.png,COVID/masks/COVID-2186.png,True,False,299,299,L,142.31,c5a7f3155e32f3a4f133bae0dd5af274
COVID-2187.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2187.png,COVID/images/COVID-2187.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2187.png,COVID/masks/COVID-2187.png,True,False,299,299,L,130.52,fab4a785040baf446e7763a57ea23250
COVID-2188.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2188.png,COVID/images/COVID-2188.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2188.png,COVID/masks/COVID-2188.png,True,False,299,299,L,146.68,2a400daecdaa34d64444eeb7b4da3ec0
COVID-2189.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2189.png,COVID/images/COVID-2189.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2189.png,COVID/masks/COVID-2189.png,True,False,299,299,L,136.05,63dde22b9fd3aae25b2d484c862ff5d7
COVID-219.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-219.png,COVID/images/COVID-219.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-219.png,COVID/masks/COVID-219.png,True,False,299,299,L,143.51,5776866e49cfa37f7add5b53349a2f15
COVID-2190.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2190.png,COVID/images/COVID-2190.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2190.png,COVID/masks/COVID-2190.png,True,False,299,299,L,147.37,548795fde836397bafd86acd60e0d3c3
COVID-2191.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2191.png,COVID/images/COVID-2191.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2191.png,COVID/masks/COVID-2191.png,True,False,299,299,L,181.91,5a22cf286eb23b3e251fb8246f358271
COVID-2192.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2192.png,COVID/images/COVID-2192.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2192.png,COVID/masks/COVID-2192.png,True,False,299,299,L,142.06,802cd512533e9f6de805642af15962a6
COVID-2193.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2193.png,COVID/images/COVID-2193.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2193.png,COVID/masks/COVID-2193.png,True,False,299,299,L,139.61,3cde2338216cd30b5206e0c4d51931b3
COVID-2194.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2194.png,COVID/images/COVID-2194.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2194.png,COVID/masks/COVID-2194.png,True,False,299,299,L,145.91,fb3f48682395e6551688ce1d4c0f0372
COVID-2195.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2195.png,COVID/images/COVID-2195.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2195.png,COVID/masks/COVID-2195.png,True,False,299,299,L,135.45,bb6a28825e913cb588108058f19a39ba
COVID-2196.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2196.png,COVID/images/COVID-2196.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2196.png,COVID/masks/COVID-2196.png,True,False,299,299,L,130.07,5097473c2be88f459448c35bd56aa08f
COVID-2197.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2197.png,COVID/images/COVID-2197.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2197.png,COVID/masks/COVID-2197.png,True,False,299,299,L,143.96,54405f075ee06a9e1479bbdfa65fc6aa
COVID-2198.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2198.png,COVID/images/COVID-2198.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2198.png,COVID/masks/COVID-2198.png,True,False,299,299,L,146.67,2ae0599dbe67942d6992b56b00aac300
COVID-2199.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2199.png,COVID/images/COVID-2199.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2199.png,COVID/masks/COVID-2199.png,True,False,299,299,L,157.12,3f7b681ff0f2fa9a37658e01369c51dd
COVID-22.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-22.png,COVID/images/COVID-22.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-22.png,COVID/masks/COVID-22.png,True,False,299,299,L,102.08,e0853b82e17fb29d08611d7671dd9132
COVID-220.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-220.png,COVID/images/COVID-220.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-220.png,COVID/masks/COVID-220.png,True,False,299,299,L,136.17,0502655828fb24e135d5c6eb6670c823
COVID-2200.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2200.png,COVID/images/COVID-2200.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2200.png,COVID/masks/COVID-2200.png,True,False,299,299,L,162.96,3beaab4526499a94cd5cb22531d14ee3
COVID-2201.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2201.png,COVID/images/COVID-2201.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2201.png,COVID/masks/COVID-2201.png,True,False,299,299,L,155.42,0942c524d606054cf9e5a4239dc57c91
COVID-2202.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2202.png,COVID/images/COVID-2202.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2202.png,COVID/masks/COVID-2202.png,True,False,299,299,L,134.81,9845ff37c0a3c14ad6ebe28a89c43877
COVID-2203.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2203.png,COVID/images/COVID-2203.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2203.png,COVID/masks/COVID-2203.png,True,False,299,299,L,151.89,3057a19ff98a8982a94418a92ac36af3
COVID-2204.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2204.png,COVID/images/COVID-2204.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2204.png,COVID/masks/COVID-2204.png,True,False,299,299,L,134.0,6105d9934ec12ea999a029ad225aed57
COVID-2205.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2205.png,COVID/images/COVID-2205.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2205.png,COVID/masks/COVID-2205.png,True,False,299,299,L,140.4,27982163eda878db52046222a613ca31
COVID-2206.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2206.png,COVID/images/COVID-2206.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2206.png,COVID/masks/COVID-2206.png,True,False,299,299,L,145.48,84c5cb2102db553438f95173eec5d9b5
COVID-2207.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2207.png,COVID/images/COVID-2207.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2207.png,COVID/masks/COVID-2207.png,True,False,299,299,L,126.94,d90a1c8217de7fe4dc5d337250fd43a9
COVID-2208.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2208.png,COVID/images/COVID-2208.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2208.png,COVID/masks/COVID-2208.png,True,False,299,299,L,123.4,044b0a5c52b2101d792e6cab48ad1a8c
COVID-2209.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2209.png,COVID/images/COVID-2209.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2209.png,COVID/masks/COVID-2209.png,True,False,299,299,L,139.41,53ac15f8996307f46bfcaab1cb54e800
COVID-221.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-221.png,COVID/images/COVID-221.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-221.png,COVID/masks/COVID-221.png,True,False,299,299,L,168.45,412b5600ccdcf260a307570d3b057887
COVID-2210.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2210.png,COVID/images/COVID-2210.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2210.png,COVID/masks/COVID-2210.png,True,False,299,299,L,125.39,aa6fa16d2d13da8a44ba9034661f0375
COVID-2211.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2211.png,COVID/images/COVID-2211.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2211.png,COVID/masks/COVID-2211.png,True,False,299,299,L,166.22,7a42eed286ed97450c46f35c689a9935
COVID-2212.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2212.png,COVID/images/COVID-2212.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2212.png,COVID/masks/COVID-2212.png,True,False,299,299,L,134.3,f3007b99087a2f4781a0e9bfabf0d242
COVID-2213.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2213.png,COVID/images/COVID-2213.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2213.png,COVID/masks/COVID-2213.png,True,False,299,299,L,153.57,9c9e89c5537e3b4013f4e4e2c5e96ebf
COVID-2214.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2214.png,COVID/images/COVID-2214.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2214.png,COVID/masks/COVID-2214.png,True,False,299,299,L,147.75,08203b2bb0e90140bd421920fa614856
COVID-2215.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2215.png,COVID/images/COVID-2215.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2215.png,COVID/masks/COVID-2215.png,True,False,299,299,L,148.86,2dca8962368ff2d47a47ffa2f92f39a4
COVID-2216.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2216.png,COVID/images/COVID-2216.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2216.png,COVID/masks/COVID-2216.png,True,False,299,299,L,144.92,38906bcfd34300df5e0bed31289397a1
COVID-2217.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2217.png,COVID/images/COVID-2217.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2217.png,COVID/masks/COVID-2217.png,True,False,299,299,L,139.36,c10ccfe8f131adc6baa64d281e193705
COVID-2218.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2218.png,COVID/images/COVID-2218.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2218.png,COVID/masks/COVID-2218.png,True,False,299,299,L,101.88,96b4304706a7f23efc7c7e62721ab7b9
COVID-2219.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2219.png,COVID/images/COVID-2219.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2219.png,COVID/masks/COVID-2219.png,True,False,299,299,L,152.92,ebb7e8e147484be1cbecdb62e7532e39
COVID-222.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-222.png,COVID/images/COVID-222.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-222.png,COVID/masks/COVID-222.png,True,False,299,299,L,139.69,486768317bf7b5d80a5f5eef6ce8504f
COVID-2220.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2220.png,COVID/images/COVID-2220.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2220.png,COVID/masks/COVID-2220.png,True,False,299,299,L,128.98,2cfdfeba6ce23674f13691e7762f2ac4
COVID-2221.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2221.png,COVID/images/COVID-2221.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2221.png,COVID/masks/COVID-2221.png,True,False,299,299,L,135.22,5a623eb2c09713c2198b982f5e72b0c5
COVID-2222.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2222.png,COVID/images/COVID-2222.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2222.png,COVID/masks/COVID-2222.png,True,False,299,299,L,145.64,412b1bb5337d1c1df35a17f1fda4882e
COVID-2223.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2223.png,COVID/images/COVID-2223.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2223.png,COVID/masks/COVID-2223.png,True,False,299,299,L,120.22,f6f4af4dc1c6fa31340593dc4a0a19f5
COVID-2224.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2224.png,COVID/images/COVID-2224.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2224.png,COVID/masks/COVID-2224.png,True,False,299,299,L,141.68,6e5b682619f51192a2953ab2198ff58a
COVID-2225.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2225.png,COVID/images/COVID-2225.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2225.png,COVID/masks/COVID-2225.png,True,False,299,299,L,142.84,847518fa7f939a687e581e57399593a3
COVID-2226.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2226.png,COVID/images/COVID-2226.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2226.png,COVID/masks/COVID-2226.png,True,False,299,299,L,122.11,66edb3855ebc878e16d7ee27040ecacc
COVID-2227.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2227.png,COVID/images/COVID-2227.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2227.png,COVID/masks/COVID-2227.png,True,False,299,299,L,148.9,aa4aff897a3daa197d7f0ffb04be3c84
COVID-2228.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2228.png,COVID/images/COVID-2228.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2228.png,COVID/masks/COVID-2228.png,True,False,299,299,L,117.65,52280a374b571f91a566fa1501ddaf6e
COVID-2229.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2229.png,COVID/images/COVID-2229.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2229.png,COVID/masks/COVID-2229.png,True,False,299,299,L,185.0,937951df6817193289f07c4e82787b71
COVID-223.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-223.png,COVID/images/COVID-223.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-223.png,COVID/masks/COVID-223.png,True,False,299,299,L,146.58,8d3b380925cb3b7e0b98d0664e54a8f1
COVID-2230.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2230.png,COVID/images/COVID-2230.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2230.png,COVID/masks/COVID-2230.png,True,False,299,299,L,135.78,e6870c541921c4166ccba3f0a15addf4
COVID-2231.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2231.png,COVID/images/COVID-2231.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2231.png,COVID/masks/COVID-2231.png,True,False,299,299,L,110.7,1792c9b5a79a16c92d6213e06f9d25c8
COVID-2232.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2232.png,COVID/images/COVID-2232.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2232.png,COVID/masks/COVID-2232.png,True,False,299,299,L,121.44,2672c87c7fd26885840d8ec0305403f2
COVID-2233.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2233.png,COVID/images/COVID-2233.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2233.png,COVID/masks/COVID-2233.png,True,False,299,299,L,139.11,60eb4953bdfe1c24baabef98a7cf47dd
COVID-2234.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2234.png,COVID/images/COVID-2234.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2234.png,COVID/masks/COVID-2234.png,True,False,299,299,L,143.76,aea779f9daaa9ef01e1d816fdce42594
COVID-2235.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2235.png,COVID/images/COVID-2235.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2235.png,COVID/masks/COVID-2235.png,True,False,299,299,L,138.78,25799eed09f5e8eaeccd8ea7643236e6
COVID-2236.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2236.png,COVID/images/COVID-2236.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2236.png,COVID/masks/COVID-2236.png,True,False,299,299,L,131.48,60a8ed887ad91a75e123a910d7790369
COVID-2237.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2237.png,COVID/images/COVID-2237.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2237.png,COVID/masks/COVID-2237.png,True,False,299,299,L,163.56,1906dc0033ea5ae9eb370875d3230fe6
COVID-2238.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2238.png,COVID/images/COVID-2238.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2238.png,COVID/masks/COVID-2238.png,True,False,299,299,L,155.42,9842d0872bbe3d14bab39747ef1382ac
COVID-2239.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2239.png,COVID/images/COVID-2239.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2239.png,COVID/masks/COVID-2239.png,True,False,299,299,L,169.43,59332dad7ab3f5beeab6248abf6cc1fe
COVID-224.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-224.png,COVID/images/COVID-224.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-224.png,COVID/masks/COVID-224.png,True,False,299,299,L,165.82,6330d5f26f9f313b6f08d23f96a5638f
COVID-2240.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2240.png,COVID/images/COVID-2240.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2240.png,COVID/masks/COVID-2240.png,True,False,299,299,L,172.16,c43c7c6d5678cac27e91c31e33fd3d54
COVID-2241.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2241.png,COVID/images/COVID-2241.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2241.png,COVID/masks/COVID-2241.png,True,False,299,299,L,153.19,c83cf5643d58d2e854df673e2901c267
COVID-2242.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2242.png,COVID/images/COVID-2242.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2242.png,COVID/masks/COVID-2242.png,True,False,299,299,L,110.47,e74fbf19e1c756728fdddf7bfe55b5d9
COVID-2243.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2243.png,COVID/images/COVID-2243.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2243.png,COVID/masks/COVID-2243.png,True,False,299,299,L,101.83,d514ba4cd3e7176c0f3ba12600f3db35
COVID-2244.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2244.png,COVID/images/COVID-2244.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2244.png,COVID/masks/COVID-2244.png,True,False,299,299,L,150.11,b31c7f03a9a66827387f8a58678b648f
COVID-2245.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2245.png,COVID/images/COVID-2245.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2245.png,COVID/masks/COVID-2245.png,True,False,299,299,L,116.19,b7082118204444e31c99d65a14922844
COVID-2246.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2246.png,COVID/images/COVID-2246.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2246.png,COVID/masks/COVID-2246.png,True,False,299,299,L,175.17,dbabb6d8805d7838f41a65ea6c9994af
COVID-2247.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2247.png,COVID/images/COVID-2247.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2247.png,COVID/masks/COVID-2247.png,True,False,299,299,L,148.91,7ad713921ea9e9ed3a44e1c15194173c
COVID-2248.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2248.png,COVID/images/COVID-2248.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2248.png,COVID/masks/COVID-2248.png,True,False,299,299,L,154.98,9c65dcff47156ade0c2fe9d0e02166b0
COVID-2249.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2249.png,COVID/images/COVID-2249.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2249.png,COVID/masks/COVID-2249.png,True,False,299,299,L,146.81,d74cb2d39b13c02ed208e82ca8267adc
COVID-225.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-225.png,COVID/images/COVID-225.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-225.png,COVID/masks/COVID-225.png,True,False,299,299,L,158.25,4536b4b94cc06ac9819cfd6a407dddc1
COVID-2250.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2250.png,COVID/images/COVID-2250.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2250.png,COVID/masks/COVID-2250.png,True,False,299,299,L,146.4,3647e0457a57003b94e11d25c5a00c2c
COVID-2251.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2251.png,COVID/images/COVID-2251.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2251.png,COVID/masks/COVID-2251.png,True,False,299,299,L,165.5,68a7447a7d2fc08c39c2784d1f2d2949
COVID-2252.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2252.png,COVID/images/COVID-2252.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2252.png,COVID/masks/COVID-2252.png,True,False,299,299,L,122.43,87e50c606fa3b666e5a4b364b66ff73a
COVID-2253.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2253.png,COVID/images/COVID-2253.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2253.png,COVID/masks/COVID-2253.png,True,False,299,299,L,137.59,1d181ce3b5b44569798fdd264a91882c
COVID-2254.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2254.png,COVID/images/COVID-2254.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2254.png,COVID/masks/COVID-2254.png,True,False,299,299,L,158.46,98c6d647f7c86a506a81740eab7a7b34
COVID-2255.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2255.png,COVID/images/COVID-2255.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2255.png,COVID/masks/COVID-2255.png,True,False,299,299,L,139.5,8c4e91eaf779f673e3e97925d0429b61
COVID-2256.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2256.png,COVID/images/COVID-2256.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2256.png,COVID/masks/COVID-2256.png,True,False,299,299,L,148.08,e678e5783327d8b7160166bba816eac9
COVID-2257.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2257.png,COVID/images/COVID-2257.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2257.png,COVID/masks/COVID-2257.png,True,False,299,299,L,138.64,25d22006073e31463340bc550900c2f9
COVID-2258.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2258.png,COVID/images/COVID-2258.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2258.png,COVID/masks/COVID-2258.png,True,False,299,299,L,135.24,c2fad859a5a8ae142f4f3b03cf82011e
COVID-2259.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2259.png,COVID/images/COVID-2259.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2259.png,COVID/masks/COVID-2259.png,True,False,299,299,L,138.78,f2bbaff48741c1a4a0774689479ba43a
COVID-226.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-226.png,COVID/images/COVID-226.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-226.png,COVID/masks/COVID-226.png,True,False,299,299,L,118.1,e2815f2b4fe712a9994d2e62bb43f91f
COVID-2260.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2260.png,COVID/images/COVID-2260.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2260.png,COVID/masks/COVID-2260.png,True,False,299,299,L,127.75,18bcecada311deddefb9648453a2440c
COVID-2261.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2261.png,COVID/images/COVID-2261.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2261.png,COVID/masks/COVID-2261.png,True,False,299,299,L,161.19,c91446e643887e4f48a999447842d80a
COVID-2262.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2262.png,COVID/images/COVID-2262.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2262.png,COVID/masks/COVID-2262.png,True,False,299,299,L,146.31,aaad813921120290e74ff57ee466016b
COVID-2263.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2263.png,COVID/images/COVID-2263.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2263.png,COVID/masks/COVID-2263.png,True,False,299,299,L,137.66,f91006bf7475da109d38bda6f75de200
COVID-2264.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2264.png,COVID/images/COVID-2264.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2264.png,COVID/masks/COVID-2264.png,True,False,299,299,L,137.3,40030f7818f015d5fed0ca6b8a4b0fc4
COVID-2265.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2265.png,COVID/images/COVID-2265.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2265.png,COVID/masks/COVID-2265.png,True,False,299,299,L,138.97,64265f5c798842b2fd8620af19f1004a
COVID-2266.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2266.png,COVID/images/COVID-2266.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2266.png,COVID/masks/COVID-2266.png,True,False,299,299,L,133.15,7a58ee80cd2fec0cac6702dea2eafc89
COVID-2267.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2267.png,COVID/images/COVID-2267.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2267.png,COVID/masks/COVID-2267.png,True,False,299,299,L,175.8,f687ac6457e5830818b3a294559ac141
COVID-2268.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2268.png,COVID/images/COVID-2268.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2268.png,COVID/masks/COVID-2268.png,True,False,299,299,L,133.76,8cab33441488ccd7b4863eb1a9aa5a0b
COVID-2269.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2269.png,COVID/images/COVID-2269.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2269.png,COVID/masks/COVID-2269.png,True,False,299,299,L,146.92,317bd05a8b3ec09802bed302b8b5e234
COVID-227.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-227.png,COVID/images/COVID-227.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-227.png,COVID/masks/COVID-227.png,True,False,299,299,L,141.46,c9b4113ceaad3c125b8a64467a41abda
COVID-2270.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2270.png,COVID/images/COVID-2270.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2270.png,COVID/masks/COVID-2270.png,True,False,299,299,L,151.41,3c48661f27e5373d39f9130c7e9bb2e7
COVID-2271.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2271.png,COVID/images/COVID-2271.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2271.png,COVID/masks/COVID-2271.png,True,False,299,299,L,126.41,f21b9e20a3b296b934af21c4b3f4e006
COVID-2272.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2272.png,COVID/images/COVID-2272.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2272.png,COVID/masks/COVID-2272.png,True,False,299,299,L,98.67,c64f207b4c6086e4e19bdb4c1ec4da48
COVID-2273.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2273.png,COVID/images/COVID-2273.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2273.png,COVID/masks/COVID-2273.png,True,False,299,299,L,142.21,cb54025c7bdf259648135e4692ea6249
COVID-2274.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2274.png,COVID/images/COVID-2274.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2274.png,COVID/masks/COVID-2274.png,True,False,299,299,L,161.08,e6ae98e39a05df5622dd391a78086be3
COVID-2275.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2275.png,COVID/images/COVID-2275.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2275.png,COVID/masks/COVID-2275.png,True,False,299,299,L,124.42,cd5e9e7e416e0fc449af3514b2225ea2
COVID-2276.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2276.png,COVID/images/COVID-2276.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2276.png,COVID/masks/COVID-2276.png,True,False,299,299,L,156.14,0efd731fb24a1fa307e14e52f75b5bff
COVID-2277.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2277.png,COVID/images/COVID-2277.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2277.png,COVID/masks/COVID-2277.png,True,False,299,299,L,156.51,3a5bc9b340d67e9676c90533d5e658b9
COVID-2278.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2278.png,COVID/images/COVID-2278.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2278.png,COVID/masks/COVID-2278.png,True,False,299,299,L,164.91,a6357c7a186164a1a5a82f9643541beb
COVID-2279.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2279.png,COVID/images/COVID-2279.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2279.png,COVID/masks/COVID-2279.png,True,False,299,299,L,135.73,329b3f00ab62fbcec2e941743e51f6c9
COVID-228.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-228.png,COVID/images/COVID-228.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-228.png,COVID/masks/COVID-228.png,True,False,299,299,L,170.83,ea4807d54e06e80bf094a186a0624ebd
COVID-2280.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2280.png,COVID/images/COVID-2280.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2280.png,COVID/masks/COVID-2280.png,True,False,299,299,L,78.09,f6fe387fc02dbffb6bcb287864d4cc77
COVID-2281.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2281.png,COVID/images/COVID-2281.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2281.png,COVID/masks/COVID-2281.png,True,False,299,299,L,151.12,46eac6bc1858fadcb85279797e927d99
COVID-2282.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2282.png,COVID/images/COVID-2282.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2282.png,COVID/masks/COVID-2282.png,True,False,299,299,L,164.29,332a09d908b7e4932a8a01ec23bdaa37
COVID-2283.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2283.png,COVID/images/COVID-2283.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2283.png,COVID/masks/COVID-2283.png,True,False,299,299,L,73.14,7ef4246a12772f70ac6140fd752dec08
COVID-2284.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2284.png,COVID/images/COVID-2284.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2284.png,COVID/masks/COVID-2284.png,True,False,299,299,L,111.46,095b9ef8c5c70626f7f827d38cd414d5
COVID-2285.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2285.png,COVID/images/COVID-2285.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2285.png,COVID/masks/COVID-2285.png,True,False,299,299,L,140.73,9c5ae5e472f3ffe805268d60fba08004
COVID-2286.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2286.png,COVID/images/COVID-2286.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2286.png,COVID/masks/COVID-2286.png,True,False,299,299,L,85.12,60481c65b66a82a6a9ef05ab2d7db20b
COVID-2287.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2287.png,COVID/images/COVID-2287.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2287.png,COVID/masks/COVID-2287.png,True,False,299,299,L,143.62,4949b9a01230a6f33124df469112391e
COVID-2288.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2288.png,COVID/images/COVID-2288.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2288.png,COVID/masks/COVID-2288.png,True,False,299,299,L,151.84,ce6f7021873fe7d29ad89772945fc3ff
COVID-2289.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2289.png,COVID/images/COVID-2289.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2289.png,COVID/masks/COVID-2289.png,True,False,299,299,L,135.56,096409b2669eeaf210b61ddf57680396
COVID-229.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-229.png,COVID/images/COVID-229.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-229.png,COVID/masks/COVID-229.png,True,False,299,299,L,123.79,99c7141f3d994cf7431bff31262bb916
COVID-2290.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2290.png,COVID/images/COVID-2290.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2290.png,COVID/masks/COVID-2290.png,True,False,299,299,L,153.61,ac89ca86eeb2dd2d94637aa5cacf2c97
COVID-2291.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2291.png,COVID/images/COVID-2291.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2291.png,COVID/masks/COVID-2291.png,True,False,299,299,L,151.62,99b41cc0fa18d487be70b449f6d111ac
COVID-2292.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2292.png,COVID/images/COVID-2292.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2292.png,COVID/masks/COVID-2292.png,True,False,299,299,L,147.14,f228cc19b7f201f81e65536fe8a76a8d
COVID-2293.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2293.png,COVID/images/COVID-2293.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2293.png,COVID/masks/COVID-2293.png,True,False,299,299,L,89.28,7ecc535a2788ac9fd00b9437f8be5dbd
COVID-2294.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2294.png,COVID/images/COVID-2294.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2294.png,COVID/masks/COVID-2294.png,True,False,299,299,L,168.65,3649f78065d944bccd37a2d3a772506f
COVID-2295.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2295.png,COVID/images/COVID-2295.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2295.png,COVID/masks/COVID-2295.png,True,False,299,299,L,133.97,ddb873324536c796561061d077e0e8c6
COVID-2296.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2296.png,COVID/images/COVID-2296.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2296.png,COVID/masks/COVID-2296.png,True,False,299,299,L,151.47,a0f1ebd0a22317e3c75a84af0a4a067b
COVID-2297.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2297.png,COVID/images/COVID-2297.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2297.png,COVID/masks/COVID-2297.png,True,False,299,299,L,132.02,0e72246f0a183c3cc58d7250aaa0eb82
COVID-2298.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2298.png,COVID/images/COVID-2298.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2298.png,COVID/masks/COVID-2298.png,True,False,299,299,L,142.85,c3a2e5faeefb8674a511a67600625c65
COVID-2299.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2299.png,COVID/images/COVID-2299.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2299.png,COVID/masks/COVID-2299.png,True,False,299,299,L,91.53,b6f1154fefd6b6128dd42a3208c74189
COVID-23.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-23.png,COVID/images/COVID-23.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-23.png,COVID/masks/COVID-23.png,True,False,299,299,L,120.35,e939ac50b30eb8f6404543f09fee252a
COVID-230.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-230.png,COVID/images/COVID-230.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-230.png,COVID/masks/COVID-230.png,True,False,299,299,L,152.75,503942315f3ebffa3b1a456a40326557
COVID-2300.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2300.png,COVID/images/COVID-2300.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2300.png,COVID/masks/COVID-2300.png,True,False,299,299,L,114.89,ff39a59d9f6f229d674aeaba332fc821
COVID-2301.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2301.png,COVID/images/COVID-2301.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2301.png,COVID/masks/COVID-2301.png,True,False,299,299,L,133.96,503e985778704d197f0cd23e515f2c2b
COVID-2302.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2302.png,COVID/images/COVID-2302.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2302.png,COVID/masks/COVID-2302.png,True,False,299,299,L,146.04,5538c777cf3dae9143ffe45bd664313c
COVID-2303.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2303.png,COVID/images/COVID-2303.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2303.png,COVID/masks/COVID-2303.png,True,False,299,299,L,159.92,4566fc19b236303bea3a72335c4740b1
COVID-2304.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2304.png,COVID/images/COVID-2304.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2304.png,COVID/masks/COVID-2304.png,True,False,299,299,L,149.36,c735890d999c57a332dd1075660d8305
COVID-2305.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2305.png,COVID/images/COVID-2305.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2305.png,COVID/masks/COVID-2305.png,True,False,299,299,L,138.12,1c4f21f7fbc43137aa09878dbb44b882
COVID-2306.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2306.png,COVID/images/COVID-2306.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2306.png,COVID/masks/COVID-2306.png,True,False,299,299,L,171.9,9a80efb694706e3f03c2ca01d71045b6
COVID-2307.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2307.png,COVID/images/COVID-2307.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2307.png,COVID/masks/COVID-2307.png,True,False,299,299,L,158.16,a19ceb27160984f0ad63f19a0a0f7c90
COVID-2308.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2308.png,COVID/images/COVID-2308.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2308.png,COVID/masks/COVID-2308.png,True,False,299,299,L,158.66,7dece96de805760493e1cca005df99ec
COVID-2309.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2309.png,COVID/images/COVID-2309.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2309.png,COVID/masks/COVID-2309.png,True,False,299,299,L,145.82,7e2681a9cc24b872d6f9b3ec6921c0f3
COVID-231.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-231.png,COVID/images/COVID-231.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-231.png,COVID/masks/COVID-231.png,True,False,299,299,L,155.79,366ad0d9be07a27c02de2345ed9c4e80
COVID-2310.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2310.png,COVID/images/COVID-2310.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2310.png,COVID/masks/COVID-2310.png,True,False,299,299,L,150.53,0c1215ac37c68f41a034208609d97dbf
COVID-2311.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2311.png,COVID/images/COVID-2311.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2311.png,COVID/masks/COVID-2311.png,True,False,299,299,L,160.68,2f7eea45c3a808d3b10c0b02ffc9dc49
COVID-2312.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2312.png,COVID/images/COVID-2312.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2312.png,COVID/masks/COVID-2312.png,True,False,299,299,L,130.09,a73e7cbb6df1e3d061ba276a3feea98a
COVID-2313.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2313.png,COVID/images/COVID-2313.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2313.png,COVID/masks/COVID-2313.png,True,False,299,299,L,180.29,9e75c3e413336ce70a222545e4ba22f9
COVID-2314.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2314.png,COVID/images/COVID-2314.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2314.png,COVID/masks/COVID-2314.png,True,False,299,299,L,173.01,c23925e0d7e9e007b72a3804f1f94853
COVID-2315.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2315.png,COVID/images/COVID-2315.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2315.png,COVID/masks/COVID-2315.png,True,False,299,299,L,153.34,6b2a7e92cdf9248b8b56ad2bf7fe6a9b
COVID-2316.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2316.png,COVID/images/COVID-2316.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2316.png,COVID/masks/COVID-2316.png,True,False,299,299,L,176.51,566ee7718d6ca5b2100daea15e012953
COVID-2317.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2317.png,COVID/images/COVID-2317.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2317.png,COVID/masks/COVID-2317.png,True,False,299,299,L,142.4,1c7bdec38da91394ed7b07df9a8c4578
COVID-2318.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2318.png,COVID/images/COVID-2318.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2318.png,COVID/masks/COVID-2318.png,True,False,299,299,L,140.85,734bfe72095f1e7f335ccd96f4bb7c5b
COVID-2319.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2319.png,COVID/images/COVID-2319.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2319.png,COVID/masks/COVID-2319.png,True,False,299,299,L,160.3,20e4a7ff34a22f8d1d600762e6e06990
COVID-232.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-232.png,COVID/images/COVID-232.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-232.png,COVID/masks/COVID-232.png,True,False,299,299,L,124.52,dd03874ebbbbb6edaf1ca7156607c410
COVID-2320.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2320.png,COVID/images/COVID-2320.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2320.png,COVID/masks/COVID-2320.png,True,False,299,299,L,164.01,0bc6c7b49c96e83308e69133ee9add5a
COVID-2321.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2321.png,COVID/images/COVID-2321.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2321.png,COVID/masks/COVID-2321.png,True,False,299,299,L,163.86,3b5c9d55818e605499863eb650cef7b9
COVID-2322.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2322.png,COVID/images/COVID-2322.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2322.png,COVID/masks/COVID-2322.png,True,False,299,299,L,139.76,64e76101db5e3dc6c3b2d197625c5151
COVID-2323.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2323.png,COVID/images/COVID-2323.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2323.png,COVID/masks/COVID-2323.png,True,False,299,299,L,149.85,e96c67600884bba6a79c84641385397d
COVID-2324.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2324.png,COVID/images/COVID-2324.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2324.png,COVID/masks/COVID-2324.png,True,False,299,299,L,54.82,754c29cf3f2a4c8f0b50714efb639d4b
COVID-2325.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2325.png,COVID/images/COVID-2325.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2325.png,COVID/masks/COVID-2325.png,True,False,299,299,L,76.36,7918cfb1c9f251140bfacefc6ddc4cd1
COVID-2326.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2326.png,COVID/images/COVID-2326.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2326.png,COVID/masks/COVID-2326.png,True,False,299,299,L,138.59,281fe5ee9d511bc939bb727980504dbc
COVID-2327.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2327.png,COVID/images/COVID-2327.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2327.png,COVID/masks/COVID-2327.png,True,False,299,299,L,185.96,31fcb2b2a9cfc68a11ee358462258b07
COVID-2328.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2328.png,COVID/images/COVID-2328.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2328.png,COVID/masks/COVID-2328.png,True,False,299,299,L,168.05,e00a3c2a221f9a055c2b1c1df4bc83e3
COVID-2329.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2329.png,COVID/images/COVID-2329.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2329.png,COVID/masks/COVID-2329.png,True,False,299,299,L,172.73,66ecb8be7321f326438eec6910276059
COVID-233.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-233.png,COVID/images/COVID-233.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-233.png,COVID/masks/COVID-233.png,True,False,299,299,L,135.51,4f9cd146fb7546641637a893ce2dfa97
COVID-2330.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2330.png,COVID/images/COVID-2330.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2330.png,COVID/masks/COVID-2330.png,True,False,299,299,L,149.69,2188be5a88f1ae27f998562e52a0e6b0
COVID-2331.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2331.png,COVID/images/COVID-2331.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2331.png,COVID/masks/COVID-2331.png,True,False,299,299,L,138.17,fe54eb642e78952b4b7908b7cdce277a
COVID-2332.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2332.png,COVID/images/COVID-2332.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2332.png,COVID/masks/COVID-2332.png,True,False,299,299,L,143.82,56a4010bcfbd1b55df5f5b56cb97a1ae
COVID-2333.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2333.png,COVID/images/COVID-2333.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2333.png,COVID/masks/COVID-2333.png,True,False,299,299,L,157.26,bbd4f7d7bcbf10fd8b14718c95eb761c
COVID-2334.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2334.png,COVID/images/COVID-2334.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2334.png,COVID/masks/COVID-2334.png,True,False,299,299,L,170.73,f185e51450adb095b23f9ac19bb28a80
COVID-2335.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2335.png,COVID/images/COVID-2335.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2335.png,COVID/masks/COVID-2335.png,True,False,299,299,L,157.63,800da262652a7b4350219244cc6d531c
COVID-2336.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2336.png,COVID/images/COVID-2336.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2336.png,COVID/masks/COVID-2336.png,True,False,299,299,L,145.51,135655148745055777aef6cc1b2059c9
COVID-2337.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2337.png,COVID/images/COVID-2337.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2337.png,COVID/masks/COVID-2337.png,True,False,299,299,L,143.39,68a8160a7116604f48974885ea3f7be7
COVID-2338.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2338.png,COVID/images/COVID-2338.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2338.png,COVID/masks/COVID-2338.png,True,False,299,299,L,144.43,4ffca159599dcbfbc3a1749ff474e78e
COVID-2339.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2339.png,COVID/images/COVID-2339.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2339.png,COVID/masks/COVID-2339.png,True,False,299,299,L,175.53,0f6e98838c88669c02c330de593731f9
COVID-234.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-234.png,COVID/images/COVID-234.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-234.png,COVID/masks/COVID-234.png,True,False,299,299,L,137.17,1d901060f57b6c25f6d2bc2106d77eb7
COVID-2340.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2340.png,COVID/images/COVID-2340.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2340.png,COVID/masks/COVID-2340.png,True,False,299,299,L,134.03,cc29bb052eb8ff5c20890c7b94cb2c96
COVID-2341.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2341.png,COVID/images/COVID-2341.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2341.png,COVID/masks/COVID-2341.png,True,False,299,299,L,105.7,24db2fd15dbb2fe774c64aa36c6274da
COVID-2342.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2342.png,COVID/images/COVID-2342.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2342.png,COVID/masks/COVID-2342.png,True,False,299,299,L,111.75,637622befa40f4658949d7e30874288d
COVID-2343.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2343.png,COVID/images/COVID-2343.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2343.png,COVID/masks/COVID-2343.png,True,False,299,299,L,148.23,7c448ad9fd92bcdabdc17019ad1ab65e
COVID-2344.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2344.png,COVID/images/COVID-2344.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2344.png,COVID/masks/COVID-2344.png,True,False,299,299,L,128.34,4863c584842d5f53fd7b375a100c8e7d
COVID-2345.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2345.png,COVID/images/COVID-2345.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2345.png,COVID/masks/COVID-2345.png,True,False,299,299,L,153.84,49326fa16dd41021c1a45ed8cbae6f82
COVID-2346.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2346.png,COVID/images/COVID-2346.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2346.png,COVID/masks/COVID-2346.png,True,False,299,299,L,157.74,2ac60b798f5dad42334dee8e59f18c3b
COVID-2347.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2347.png,COVID/images/COVID-2347.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2347.png,COVID/masks/COVID-2347.png,True,False,299,299,L,136.18,8bfa66c6bfdd8ca1e8e6756c4368e577
COVID-2348.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2348.png,COVID/images/COVID-2348.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2348.png,COVID/masks/COVID-2348.png,True,False,299,299,L,111.9,b7a2bcc72cef767563d9afab2041cbdd
COVID-2349.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2349.png,COVID/images/COVID-2349.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2349.png,COVID/masks/COVID-2349.png,True,False,299,299,L,155.22,d3af0e205c1f6b6020a40ed87d032d1e
COVID-235.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-235.png,COVID/images/COVID-235.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-235.png,COVID/masks/COVID-235.png,True,False,299,299,L,134.39,72c150a799a99b3f1dd9af334ad6b098
COVID-2350.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2350.png,COVID/images/COVID-2350.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2350.png,COVID/masks/COVID-2350.png,True,False,299,299,L,121.92,adea5b7c7c8292adab214cee8f7051be
COVID-2351.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2351.png,COVID/images/COVID-2351.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2351.png,COVID/masks/COVID-2351.png,True,False,299,299,L,132.1,26866ae8bbc872180d9e686134e2a0f9
COVID-2352.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2352.png,COVID/images/COVID-2352.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2352.png,COVID/masks/COVID-2352.png,True,False,299,299,L,126.91,70ab9b1fb47165d2372a61ea0f47ddb0
COVID-2353.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2353.png,COVID/images/COVID-2353.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2353.png,COVID/masks/COVID-2353.png,True,False,299,299,L,172.56,cbd7b0f7d2c8218793428eb45b21632f
COVID-2354.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2354.png,COVID/images/COVID-2354.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2354.png,COVID/masks/COVID-2354.png,True,False,299,299,L,131.83,7f39cac9e522069d9409b45ab25f3d0a
COVID-2355.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2355.png,COVID/images/COVID-2355.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2355.png,COVID/masks/COVID-2355.png,True,False,299,299,L,62.71,ea39be72610db5573f257be8ed428ef9
COVID-2356.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2356.png,COVID/images/COVID-2356.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2356.png,COVID/masks/COVID-2356.png,True,False,299,299,L,129.54,3f99b9a8f974bd778e925a0aebfc59aa
COVID-2357.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2357.png,COVID/images/COVID-2357.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2357.png,COVID/masks/COVID-2357.png,True,False,299,299,L,157.09,3de036d165cfd557c26f9a8a8a3bf019
COVID-2358.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2358.png,COVID/images/COVID-2358.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2358.png,COVID/masks/COVID-2358.png,True,False,299,299,L,109.39,87ad7df164fd7c3df770bc2d7e2d226f
COVID-2359.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2359.png,COVID/images/COVID-2359.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2359.png,COVID/masks/COVID-2359.png,True,False,299,299,L,135.07,3cb1131b122380ad6b28e0c7e02b66d4
COVID-236.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-236.png,COVID/images/COVID-236.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-236.png,COVID/masks/COVID-236.png,True,False,299,299,L,169.75,5744352bc4e916dcc0238182a7b1e19a
COVID-2360.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2360.png,COVID/images/COVID-2360.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2360.png,COVID/masks/COVID-2360.png,True,False,299,299,L,157.45,184508fb4bb972f79222f59e82b320db
COVID-2361.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2361.png,COVID/images/COVID-2361.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2361.png,COVID/masks/COVID-2361.png,True,False,299,299,L,152.98,4743cbad2b8621d2dafc333ef88520d7
COVID-2362.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2362.png,COVID/images/COVID-2362.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2362.png,COVID/masks/COVID-2362.png,True,False,299,299,L,187.14,327c8ca3a8acf4e8b5042c7fcba3727c
COVID-2363.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2363.png,COVID/images/COVID-2363.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2363.png,COVID/masks/COVID-2363.png,True,False,299,299,L,113.26,cef14db08d5769773238105850f1e7e9
COVID-2364.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2364.png,COVID/images/COVID-2364.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2364.png,COVID/masks/COVID-2364.png,True,False,299,299,L,131.05,479037194454b5a4db2ff9d02182c527
COVID-2365.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2365.png,COVID/images/COVID-2365.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2365.png,COVID/masks/COVID-2365.png,True,False,299,299,L,152.55,b95726d12d47e074de8eee601b9030df
COVID-2366.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2366.png,COVID/images/COVID-2366.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2366.png,COVID/masks/COVID-2366.png,True,False,299,299,L,115.77,07134df437b42fc02b5842aa9a043a7e
COVID-2367.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2367.png,COVID/images/COVID-2367.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2367.png,COVID/masks/COVID-2367.png,True,False,299,299,L,141.76,8cf4b5fd69430ad7b69b58f4263a291d
COVID-2368.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2368.png,COVID/images/COVID-2368.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2368.png,COVID/masks/COVID-2368.png,True,False,299,299,L,134.14,6d2ec2bb1ff46dc83e7c9a43592da507
COVID-2369.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2369.png,COVID/images/COVID-2369.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2369.png,COVID/masks/COVID-2369.png,True,False,299,299,L,149.01,90ce4f99213e65945f87258f5d81aa3b
COVID-237.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-237.png,COVID/images/COVID-237.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-237.png,COVID/masks/COVID-237.png,True,False,299,299,L,135.66,ff64d0a766c4ddbd005b63c6fc17ce28
COVID-2370.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2370.png,COVID/images/COVID-2370.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2370.png,COVID/masks/COVID-2370.png,True,False,299,299,L,148.38,27fbdd90f08b9034ebe2a1aa93339c62
COVID-2371.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2371.png,COVID/images/COVID-2371.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2371.png,COVID/masks/COVID-2371.png,True,False,299,299,L,143.22,c00c26c3ecc90ce55a24a506fee5b144
COVID-2372.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2372.png,COVID/images/COVID-2372.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2372.png,COVID/masks/COVID-2372.png,True,False,299,299,L,149.51,2b8028dfb1c9d449525bb472b1172ef0
COVID-2373.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2373.png,COVID/images/COVID-2373.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2373.png,COVID/masks/COVID-2373.png,True,False,299,299,L,132.16,430c50f36d15914432abd304ce2ce6e2
COVID-2374.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2374.png,COVID/images/COVID-2374.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2374.png,COVID/masks/COVID-2374.png,True,False,299,299,L,76.92,e9645be750c8665459243cdf995136cc
COVID-2375.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2375.png,COVID/images/COVID-2375.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2375.png,COVID/masks/COVID-2375.png,True,False,299,299,L,139.44,396bc893f4aa2576e809937fe221f1fe
COVID-2376.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2376.png,COVID/images/COVID-2376.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2376.png,COVID/masks/COVID-2376.png,True,False,299,299,L,140.57,fc23d52a8f6c979789b9ee36110daebb
COVID-2377.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2377.png,COVID/images/COVID-2377.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2377.png,COVID/masks/COVID-2377.png,True,False,299,299,L,144.98,e04a02808833f025de523fad58b9a78e
COVID-2378.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2378.png,COVID/images/COVID-2378.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2378.png,COVID/masks/COVID-2378.png,True,False,299,299,L,156.7,708fb7cd54067e35c7b8b34667d28847
COVID-2379.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2379.png,COVID/images/COVID-2379.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2379.png,COVID/masks/COVID-2379.png,True,False,299,299,L,169.94,937a513b4f0d699a26f0600a69c2571f
COVID-238.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-238.png,COVID/images/COVID-238.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-238.png,COVID/masks/COVID-238.png,True,False,299,299,L,151.27,8351dbdf7d51c3863564f2cb7d2d1545
COVID-2380.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2380.png,COVID/images/COVID-2380.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2380.png,COVID/masks/COVID-2380.png,True,False,299,299,L,189.12,e35bebf5257e0d1cfb68062b017b2ae6
COVID-2381.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2381.png,COVID/images/COVID-2381.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2381.png,COVID/masks/COVID-2381.png,True,False,299,299,L,155.51,f4e3be87c799b50676b267fdfbe18afd
COVID-2382.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2382.png,COVID/images/COVID-2382.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2382.png,COVID/masks/COVID-2382.png,True,False,299,299,L,152.97,7804ef50e4dd37545f9d865020075412
COVID-2383.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2383.png,COVID/images/COVID-2383.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2383.png,COVID/masks/COVID-2383.png,True,False,299,299,L,159.2,81b3dcf35137687cb1e35ea59eacd22f
COVID-2384.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2384.png,COVID/images/COVID-2384.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2384.png,COVID/masks/COVID-2384.png,True,False,299,299,L,146.71,406da3c5cec6e2ed4ba2db031105fbc7
COVID-2385.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2385.png,COVID/images/COVID-2385.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2385.png,COVID/masks/COVID-2385.png,True,False,299,299,L,159.08,24529157aee60fe2aed8658a08164dab
COVID-2386.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2386.png,COVID/images/COVID-2386.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2386.png,COVID/masks/COVID-2386.png,True,False,299,299,L,172.61,6006a64a089a1cad30a581ad76ec1b2a
COVID-2387.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2387.png,COVID/images/COVID-2387.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2387.png,COVID/masks/COVID-2387.png,True,False,299,299,L,160.34,201fd3947d2538e70fc760dcfb9ffe14
COVID-2388.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2388.png,COVID/images/COVID-2388.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2388.png,COVID/masks/COVID-2388.png,True,False,299,299,L,153.04,fd7305b1553a433ee47b4e24b82a28b0
COVID-2389.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2389.png,COVID/images/COVID-2389.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2389.png,COVID/masks/COVID-2389.png,True,False,299,299,L,154.58,e4e78cceb41a8fe3e9e7def802e39c7c
COVID-239.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-239.png,COVID/images/COVID-239.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-239.png,COVID/masks/COVID-239.png,True,False,299,299,L,188.57,9b4006c87456ae636291c812bc279731
COVID-2390.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2390.png,COVID/images/COVID-2390.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2390.png,COVID/masks/COVID-2390.png,True,False,299,299,L,143.91,524d80bd76a3bf50a2ebcaf2491bde18
COVID-2391.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2391.png,COVID/images/COVID-2391.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2391.png,COVID/masks/COVID-2391.png,True,False,299,299,L,141.19,ed58688922537ebb1bb447c106e1d278
COVID-2392.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2392.png,COVID/images/COVID-2392.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2392.png,COVID/masks/COVID-2392.png,True,False,299,299,L,158.86,a1f60403f41d5c29dcd5552fe4d058bb
COVID-2393.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2393.png,COVID/images/COVID-2393.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2393.png,COVID/masks/COVID-2393.png,True,False,299,299,L,146.06,70232461350c123e2316c97662700e58
COVID-2394.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2394.png,COVID/images/COVID-2394.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2394.png,COVID/masks/COVID-2394.png,True,False,299,299,L,182.16,f4aa52e3cb39e0636873fca344803290
COVID-2395.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2395.png,COVID/images/COVID-2395.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2395.png,COVID/masks/COVID-2395.png,True,False,299,299,L,148.1,b78a3a3d3d0858cf609adabbb890ca42
COVID-2396.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2396.png,COVID/images/COVID-2396.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2396.png,COVID/masks/COVID-2396.png,True,False,299,299,L,150.42,5fc7877d2d1fe9c0bd827a8ccef12eb4
COVID-2397.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2397.png,COVID/images/COVID-2397.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2397.png,COVID/masks/COVID-2397.png,True,False,299,299,L,154.3,db8d7a13883e1591c19f794c95bf6048
COVID-2398.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2398.png,COVID/images/COVID-2398.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2398.png,COVID/masks/COVID-2398.png,True,False,299,299,L,134.36,2f34a28a604b311bdd4dbd6a821a8768
COVID-2399.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2399.png,COVID/images/COVID-2399.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2399.png,COVID/masks/COVID-2399.png,True,False,299,299,L,171.77,c9b412c0ddc653c376c9506f47aa40f2
COVID-24.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-24.png,COVID/images/COVID-24.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-24.png,COVID/masks/COVID-24.png,True,False,299,299,L,148.58,ac7298b9704cf87442eb838b2e4c3c18
COVID-240.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-240.png,COVID/images/COVID-240.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-240.png,COVID/masks/COVID-240.png,True,False,299,299,L,85.8,01daa8679543339f49f6d3320a01be8a
COVID-2400.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2400.png,COVID/images/COVID-2400.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2400.png,COVID/masks/COVID-2400.png,True,False,299,299,L,132.03,2aac3b64cc9c0886993500381d052575
COVID-2401.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2401.png,COVID/images/COVID-2401.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2401.png,COVID/masks/COVID-2401.png,True,False,299,299,L,122.59,389e8095d2f9bd20f26e16e29a32eee7
COVID-2402.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2402.png,COVID/images/COVID-2402.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2402.png,COVID/masks/COVID-2402.png,True,False,299,299,L,112.23,9b56990777fea5983ca6a82cbef174e6
COVID-2403.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2403.png,COVID/images/COVID-2403.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2403.png,COVID/masks/COVID-2403.png,True,False,299,299,L,127.54,26e6a4ef09cae8edee68df56249cec20
COVID-2404.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2404.png,COVID/images/COVID-2404.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2404.png,COVID/masks/COVID-2404.png,True,False,299,299,L,158.73,4dcba1572dc0bf6a3014dcc01ac9bc69
COVID-2405.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2405.png,COVID/images/COVID-2405.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2405.png,COVID/masks/COVID-2405.png,True,False,299,299,L,152.33,db84b5cd716a081dd799902b71e41380
COVID-2406.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2406.png,COVID/images/COVID-2406.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2406.png,COVID/masks/COVID-2406.png,True,False,299,299,L,165.5,6a5c5e6a52043965d279428b58e441df
COVID-2407.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2407.png,COVID/images/COVID-2407.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2407.png,COVID/masks/COVID-2407.png,True,False,299,299,L,141.01,fa3828c20bfc36d7751912e6033482c9
COVID-2408.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2408.png,COVID/images/COVID-2408.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2408.png,COVID/masks/COVID-2408.png,True,False,299,299,L,180.09,ef3efd2fd761e61b2d4b05596646a495
COVID-2409.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2409.png,COVID/images/COVID-2409.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2409.png,COVID/masks/COVID-2409.png,True,False,299,299,L,167.26,f8b3c2218ca7710bf7d7a832edb2813e
COVID-241.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-241.png,COVID/images/COVID-241.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-241.png,COVID/masks/COVID-241.png,True,False,299,299,L,158.41,283c2b06960c7b8d07a67170d711b6be
COVID-2410.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2410.png,COVID/images/COVID-2410.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2410.png,COVID/masks/COVID-2410.png,True,False,299,299,L,134.6,39bf099c82f57dd6d570c741163d0a94
COVID-2411.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2411.png,COVID/images/COVID-2411.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2411.png,COVID/masks/COVID-2411.png,True,False,299,299,L,98.83,9e978642d96ddc0aa9cf048b92d00005
COVID-2412.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2412.png,COVID/images/COVID-2412.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2412.png,COVID/masks/COVID-2412.png,True,False,299,299,L,164.2,694529b9157806783e3ed75cb6e61222
COVID-2413.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2413.png,COVID/images/COVID-2413.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2413.png,COVID/masks/COVID-2413.png,True,False,299,299,L,89.94,bc8833297dd6b83530b3b8dfaa4c7a62
COVID-2414.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2414.png,COVID/images/COVID-2414.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2414.png,COVID/masks/COVID-2414.png,True,False,299,299,L,110.38,93030915c37ece766a21502e32013980
COVID-2415.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2415.png,COVID/images/COVID-2415.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2415.png,COVID/masks/COVID-2415.png,True,False,299,299,L,152.46,027ab20c974b344c597470b6225b776f
COVID-2416.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2416.png,COVID/images/COVID-2416.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2416.png,COVID/masks/COVID-2416.png,True,False,299,299,L,111.83,0b287468e4b1df0d234306f25b1c4b80
COVID-2417.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2417.png,COVID/images/COVID-2417.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2417.png,COVID/masks/COVID-2417.png,True,False,299,299,L,129.73,ff584375bff797f0b0c3502a7b7fad75
COVID-2418.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2418.png,COVID/images/COVID-2418.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2418.png,COVID/masks/COVID-2418.png,True,False,299,299,L,111.83,0b287468e4b1df0d234306f25b1c4b80
COVID-2419.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2419.png,COVID/images/COVID-2419.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2419.png,COVID/masks/COVID-2419.png,True,False,299,299,L,132.12,0cc538ef316d40f83bdbc873b9b9d491
COVID-242.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-242.png,COVID/images/COVID-242.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-242.png,COVID/masks/COVID-242.png,True,False,299,299,L,62.29,6957f0ecb6c13646ead3a47109550e09
COVID-2420.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2420.png,COVID/images/COVID-2420.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2420.png,COVID/masks/COVID-2420.png,True,False,299,299,L,132.12,4e013270650f939b3b50c37d1d217337
COVID-2421.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2421.png,COVID/images/COVID-2421.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2421.png,COVID/masks/COVID-2421.png,True,False,299,299,L,132.12,4e013270650f939b3b50c37d1d217337
COVID-2422.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2422.png,COVID/images/COVID-2422.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2422.png,COVID/masks/COVID-2422.png,True,False,299,299,L,132.08,6927bed5f4e8c43b3df6e4d36e31e01f
COVID-2423.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2423.png,COVID/images/COVID-2423.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2423.png,COVID/masks/COVID-2423.png,True,False,299,299,L,132.08,6927bed5f4e8c43b3df6e4d36e31e01f
COVID-2424.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2424.png,COVID/images/COVID-2424.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2424.png,COVID/masks/COVID-2424.png,True,False,299,299,L,113.13,57f9368fc5a40fa08e0e02e4b068d3aa
COVID-2425.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2425.png,COVID/images/COVID-2425.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2425.png,COVID/masks/COVID-2425.png,True,False,299,299,L,119.51,cb94459cdf0646886267d58e50c4ead3
COVID-2426.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2426.png,COVID/images/COVID-2426.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2426.png,COVID/masks/COVID-2426.png,True,False,299,299,L,142.26,07b106a3e1e4b1c2cd7a7e7e27bf018d
COVID-2427.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2427.png,COVID/images/COVID-2427.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2427.png,COVID/masks/COVID-2427.png,True,False,299,299,L,115.25,ff4ec257acc2b9ff6e513b1f6d732ff2
COVID-2428.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2428.png,COVID/images/COVID-2428.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2428.png,COVID/masks/COVID-2428.png,True,False,299,299,L,99.08,acaa22d62ff1925b3a3a2975e8822c99
COVID-2429.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2429.png,COVID/images/COVID-2429.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2429.png,COVID/masks/COVID-2429.png,True,False,299,299,L,140.23,a1d7f0d0da02d365d1cebbd22e0af0b0
COVID-243.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-243.png,COVID/images/COVID-243.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-243.png,COVID/masks/COVID-243.png,True,False,299,299,L,139.58,2ff0452b98e5bc20565c464d3b4b2615
COVID-2430.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2430.png,COVID/images/COVID-2430.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2430.png,COVID/masks/COVID-2430.png,True,False,299,299,L,135.9,6be539354ba81ac2663bd73a635c7ec0
COVID-2431.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2431.png,COVID/images/COVID-2431.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2431.png,COVID/masks/COVID-2431.png,True,False,299,299,L,144.05,6a0c74f1833da20a3ff39c6a5b48df82
COVID-2432.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2432.png,COVID/images/COVID-2432.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2432.png,COVID/masks/COVID-2432.png,True,False,299,299,L,144.05,cb57413d86da8811f3c495d81743d19f
COVID-2433.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2433.png,COVID/images/COVID-2433.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2433.png,COVID/masks/COVID-2433.png,True,False,299,299,L,131.81,6718b126160b0ebb949e1ad933bd96b9
COVID-2434.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2434.png,COVID/images/COVID-2434.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2434.png,COVID/masks/COVID-2434.png,True,False,299,299,L,134.6,64e6f324f5e352acd7f4ae435fe0cd9c
COVID-2435.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2435.png,COVID/images/COVID-2435.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2435.png,COVID/masks/COVID-2435.png,True,False,299,299,L,160.62,8e22c11fcfb0b0b7d58ceb6efa2ded95
COVID-2436.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2436.png,COVID/images/COVID-2436.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2436.png,COVID/masks/COVID-2436.png,True,False,299,299,L,157.38,3732eb32b5ab423620d5065ae1d982dc
COVID-2437.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2437.png,COVID/images/COVID-2437.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2437.png,COVID/masks/COVID-2437.png,True,False,299,299,L,136.34,aff96a8df5f8107aa781a1555b09b08e
COVID-2438.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2438.png,COVID/images/COVID-2438.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2438.png,COVID/masks/COVID-2438.png,True,False,299,299,L,137.2,dc73281e0d8c0bfbc91dd01e2d4bbb1e
COVID-2439.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2439.png,COVID/images/COVID-2439.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2439.png,COVID/masks/COVID-2439.png,True,False,299,299,L,150.47,f960199bffc6982c58aabbddad65fb8a
COVID-244.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-244.png,COVID/images/COVID-244.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-244.png,COVID/masks/COVID-244.png,True,False,299,299,L,157.24,c3d0896dbb651df009018bebcc01957f
COVID-2440.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2440.png,COVID/images/COVID-2440.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2440.png,COVID/masks/COVID-2440.png,True,False,299,299,L,135.85,115c114255242a2707a6b1a19bc8a4ab
COVID-2441.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2441.png,COVID/images/COVID-2441.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2441.png,COVID/masks/COVID-2441.png,True,False,299,299,L,132.6,f8f3d3477837961bf5a79abbfb2aa5bd
COVID-2442.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2442.png,COVID/images/COVID-2442.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2442.png,COVID/masks/COVID-2442.png,True,False,299,299,L,129.73,f974a1b0586b47999d266576c9383634
COVID-2443.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2443.png,COVID/images/COVID-2443.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2443.png,COVID/masks/COVID-2443.png,True,False,299,299,L,110.03,00c6eef33608a9b3d8ff810b0d40d979
COVID-2444.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2444.png,COVID/images/COVID-2444.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2444.png,COVID/masks/COVID-2444.png,True,False,299,299,L,136.03,5f41b76ca40c5c2d38e60a90687fdcd0
COVID-2445.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2445.png,COVID/images/COVID-2445.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2445.png,COVID/masks/COVID-2445.png,True,False,299,299,L,135.88,3a033c10e012b6f3b9be1a9e0f772169
COVID-2446.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2446.png,COVID/images/COVID-2446.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2446.png,COVID/masks/COVID-2446.png,True,False,299,299,L,137.48,500e0f094e9c9b90087994692619eb44
COVID-2447.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2447.png,COVID/images/COVID-2447.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2447.png,COVID/masks/COVID-2447.png,True,False,299,299,L,139.74,cbd71ce705ceac408e417ef21867d8dd
COVID-2448.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2448.png,COVID/images/COVID-2448.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2448.png,COVID/masks/COVID-2448.png,True,False,299,299,L,105.25,0fe59698133b098b42917ab8310db4b9
COVID-2449.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2449.png,COVID/images/COVID-2449.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2449.png,COVID/masks/COVID-2449.png,True,False,299,299,L,132.62,74182e7d0efd12cf90093434e12cf7ef
COVID-245.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-245.png,COVID/images/COVID-245.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-245.png,COVID/masks/COVID-245.png,True,False,299,299,L,159.4,ad32d686ab288d35e580dd8b33820df2
COVID-2450.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2450.png,COVID/images/COVID-2450.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2450.png,COVID/masks/COVID-2450.png,True,False,299,299,L,159.23,375a4dc43c3ee41d962b21002626091a
COVID-2451.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2451.png,COVID/images/COVID-2451.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2451.png,COVID/masks/COVID-2451.png,True,False,299,299,L,79.29,4476024937a90cd8c4c35106b1ae52db
COVID-2452.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2452.png,COVID/images/COVID-2452.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2452.png,COVID/masks/COVID-2452.png,True,False,299,299,L,133.45,1779f3137ce0aa2d3ce0513963c5a842
COVID-2453.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2453.png,COVID/images/COVID-2453.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2453.png,COVID/masks/COVID-2453.png,True,False,299,299,L,133.94,e4198f010ce604a0155f209e7d85933c
COVID-2454.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2454.png,COVID/images/COVID-2454.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2454.png,COVID/masks/COVID-2454.png,True,False,299,299,L,131.24,1ab7e7d5bb5ee181449c817d757df55f
COVID-2455.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2455.png,COVID/images/COVID-2455.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2455.png,COVID/masks/COVID-2455.png,True,False,299,299,L,68.89,524b2f9b2dc8e267c5fd6b05eead2c68
COVID-2456.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2456.png,COVID/images/COVID-2456.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2456.png,COVID/masks/COVID-2456.png,True,False,299,299,L,175.96,149cdbe0e6a1f26b36397bf0c8f5a750
COVID-2457.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2457.png,COVID/images/COVID-2457.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2457.png,COVID/masks/COVID-2457.png,True,False,299,299,L,165.46,acd2991b965ca73f621ec032e2d86d69
COVID-2458.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2458.png,COVID/images/COVID-2458.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2458.png,COVID/masks/COVID-2458.png,True,False,299,299,L,162.16,21f8a38e311cd46fdc3df0e118c0892d
COVID-2459.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2459.png,COVID/images/COVID-2459.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2459.png,COVID/masks/COVID-2459.png,True,False,299,299,L,162.16,21f8a38e311cd46fdc3df0e118c0892d
COVID-246.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-246.png,COVID/images/COVID-246.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-246.png,COVID/masks/COVID-246.png,True,False,299,299,L,146.02,d2a409269d5c3e26883f722e5e6c8cb1
COVID-2460.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2460.png,COVID/images/COVID-2460.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2460.png,COVID/masks/COVID-2460.png,True,False,299,299,L,149.63,0e16fc93a6ca132be19abd1b34e37d1c
COVID-2461.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2461.png,COVID/images/COVID-2461.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2461.png,COVID/masks/COVID-2461.png,True,False,299,299,L,125.02,bfd624cb2e5da97b24c322e17ed36705
COVID-2462.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2462.png,COVID/images/COVID-2462.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2462.png,COVID/masks/COVID-2462.png,True,False,299,299,L,157.06,f9c54a3d4b9e70a559f2b9dc218c6297
COVID-2463.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2463.png,COVID/images/COVID-2463.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2463.png,COVID/masks/COVID-2463.png,True,False,299,299,L,159.97,1ced25be8f4775f89e6898ad599ed877
COVID-2464.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2464.png,COVID/images/COVID-2464.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2464.png,COVID/masks/COVID-2464.png,True,False,299,299,L,122.87,e69663e7e2fad5532f029087e2a657b0
COVID-2465.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2465.png,COVID/images/COVID-2465.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2465.png,COVID/masks/COVID-2465.png,True,False,299,299,L,43.57,7282f714d9d928e266416c31c1d8bacb
COVID-2466.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2466.png,COVID/images/COVID-2466.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2466.png,COVID/masks/COVID-2466.png,True,False,299,299,L,57.71,1038dcf3b145217afec63e0f05dd3607
COVID-2467.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2467.png,COVID/images/COVID-2467.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2467.png,COVID/masks/COVID-2467.png,True,False,299,299,L,163.02,4e6cabb64a31604085cdd49cdef83b3a
COVID-2468.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2468.png,COVID/images/COVID-2468.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2468.png,COVID/masks/COVID-2468.png,True,False,299,299,L,62.41,36877803f8f4f213a8c3198bc940bbd4
COVID-2469.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2469.png,COVID/images/COVID-2469.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2469.png,COVID/masks/COVID-2469.png,True,False,299,299,L,167.2,5cb97d4d9ddc53e1eec5ba9787d4506b
COVID-247.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-247.png,COVID/images/COVID-247.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-247.png,COVID/masks/COVID-247.png,True,False,299,299,L,136.27,f7e6e500001e97ebc1931176a1a9f167
COVID-2470.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2470.png,COVID/images/COVID-2470.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2470.png,COVID/masks/COVID-2470.png,True,False,299,299,L,145.43,70928a44419ec36b849954a187cf40fd
COVID-2471.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2471.png,COVID/images/COVID-2471.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2471.png,COVID/masks/COVID-2471.png,True,False,299,299,L,154.92,2bc182349b1c95e9cf360ca26a6bcdc9
COVID-2472.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2472.png,COVID/images/COVID-2472.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2472.png,COVID/masks/COVID-2472.png,True,False,299,299,L,207.28,a389ca84e1a909cf3c18f674da28d159
COVID-2473.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2473.png,COVID/images/COVID-2473.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2473.png,COVID/masks/COVID-2473.png,True,False,299,299,L,152.94,540274fdfe2eb81cb1d530f1a7db5a20
COVID-2474.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2474.png,COVID/images/COVID-2474.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2474.png,COVID/masks/COVID-2474.png,True,False,299,299,L,154.64,ab884a65193f7a79ab45f04651ccd79d
COVID-2475.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2475.png,COVID/images/COVID-2475.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2475.png,COVID/masks/COVID-2475.png,True,False,299,299,L,154.75,dde4dea1c2ff228d7366575746b767b3
COVID-2476.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2476.png,COVID/images/COVID-2476.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2476.png,COVID/masks/COVID-2476.png,True,False,299,299,L,146.44,9a84117d5ccafad0d8a3b6502f3c5288
COVID-2477.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2477.png,COVID/images/COVID-2477.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2477.png,COVID/masks/COVID-2477.png,True,False,299,299,L,186.62,f0e3cbfb11726d4b829dd4f7f8cd45f1
COVID-2478.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2478.png,COVID/images/COVID-2478.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2478.png,COVID/masks/COVID-2478.png,True,False,299,299,L,172.21,ef8e2e533fb2e1514837651805d49f86
COVID-2479.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2479.png,COVID/images/COVID-2479.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2479.png,COVID/masks/COVID-2479.png,True,False,299,299,L,155.96,621498e8f895f87bd7c0a505d2075db6
COVID-248.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-248.png,COVID/images/COVID-248.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-248.png,COVID/masks/COVID-248.png,True,False,299,299,L,159.35,2ada1bd7b96e04ca1c7e69619cf820eb
COVID-2480.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2480.png,COVID/images/COVID-2480.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2480.png,COVID/masks/COVID-2480.png,True,False,299,299,L,190.83,ed0f5083b1220f2fce02e3e946002556
COVID-2481.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2481.png,COVID/images/COVID-2481.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2481.png,COVID/masks/COVID-2481.png,True,False,299,299,L,139.0,b35245d3da641cb5305e82ffe0cf3b35
COVID-2482.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2482.png,COVID/images/COVID-2482.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2482.png,COVID/masks/COVID-2482.png,True,False,299,299,L,83.23,a1fabf50cb3c917d465cc8d0f04f47f3
COVID-2483.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2483.png,COVID/images/COVID-2483.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2483.png,COVID/masks/COVID-2483.png,True,False,299,299,L,95.64,199df7766179625659cab3310fe42061
COVID-2484.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2484.png,COVID/images/COVID-2484.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2484.png,COVID/masks/COVID-2484.png,True,False,299,299,L,115.57,c045030a2fee322cc23570e69411cc0e
COVID-2485.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2485.png,COVID/images/COVID-2485.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2485.png,COVID/masks/COVID-2485.png,True,False,299,299,L,115.51,7f23a71cfde035dde5723c6f1dd1848e
COVID-2486.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2486.png,COVID/images/COVID-2486.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2486.png,COVID/masks/COVID-2486.png,True,False,299,299,L,164.7,79b99eeb721f8d96750693949f382114
COVID-2487.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2487.png,COVID/images/COVID-2487.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2487.png,COVID/masks/COVID-2487.png,True,False,299,299,L,67.77,3946510f3a9a7d1e05213b25d60131be
COVID-2488.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2488.png,COVID/images/COVID-2488.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2488.png,COVID/masks/COVID-2488.png,True,False,299,299,L,126.66,f6f831cf838b6a8662c020fa2c2f72e7
COVID-2489.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2489.png,COVID/images/COVID-2489.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2489.png,COVID/masks/COVID-2489.png,True,False,299,299,L,141.37,e4fbc31de8f2d9317c90a0a606935836
COVID-249.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-249.png,COVID/images/COVID-249.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-249.png,COVID/masks/COVID-249.png,True,False,299,299,L,139.0,b199af3748b2e19ec8a76ffe4eb9460b
COVID-2490.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2490.png,COVID/images/COVID-2490.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2490.png,COVID/masks/COVID-2490.png,True,False,299,299,L,135.02,36d3abafc4039c2ff16d17c40ea74869
COVID-2491.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2491.png,COVID/images/COVID-2491.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2491.png,COVID/masks/COVID-2491.png,True,False,299,299,L,138.51,ca1132588172ea2ec7bc9aa462d494ab
COVID-2492.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2492.png,COVID/images/COVID-2492.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2492.png,COVID/masks/COVID-2492.png,True,False,299,299,L,154.3,3ba722d1d93f3f80763c46b8c7af5599
COVID-2493.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2493.png,COVID/images/COVID-2493.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2493.png,COVID/masks/COVID-2493.png,True,False,299,299,L,154.3,3ba722d1d93f3f80763c46b8c7af5599
COVID-2494.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2494.png,COVID/images/COVID-2494.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2494.png,COVID/masks/COVID-2494.png,True,False,299,299,L,139.08,3429bd5859b2547aeb44b46acc0bdd06
COVID-2495.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2495.png,COVID/images/COVID-2495.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2495.png,COVID/masks/COVID-2495.png,True,False,299,299,L,139.34,aed69d23aa59b57b4b47c1d28c9df2a4
COVID-2496.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2496.png,COVID/images/COVID-2496.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2496.png,COVID/masks/COVID-2496.png,True,False,299,299,L,197.15,742c5419a54cc5ac85fe857a06dcd5d2
COVID-2497.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2497.png,COVID/images/COVID-2497.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2497.png,COVID/masks/COVID-2497.png,True,False,299,299,L,125.61,7ae4d49dc2ead766f3ad9be6d99e178a
COVID-2498.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2498.png,COVID/images/COVID-2498.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2498.png,COVID/masks/COVID-2498.png,True,False,299,299,L,125.17,1c38aef563f9cf6fb4151abc7d5d937b
COVID-2499.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2499.png,COVID/images/COVID-2499.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2499.png,COVID/masks/COVID-2499.png,True,False,299,299,L,125.84,50d84478a466167ed05c9ec6cbf5dbed
COVID-25.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-25.png,COVID/images/COVID-25.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-25.png,COVID/masks/COVID-25.png,True,False,299,299,L,128.78,16f065a3d3b6b66a07fbeb92a187b3c3
COVID-250.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-250.png,COVID/images/COVID-250.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-250.png,COVID/masks/COVID-250.png,True,False,299,299,L,117.97,953fcf78eef0297e36c9a35fbf3a375f
COVID-2500.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2500.png,COVID/images/COVID-2500.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2500.png,COVID/masks/COVID-2500.png,True,False,299,299,L,133.95,014aa1ca8db505e709bb16fb5d7fd79e
COVID-2501.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2501.png,COVID/images/COVID-2501.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2501.png,COVID/masks/COVID-2501.png,True,False,299,299,L,145.65,f03e3011bf62bb715797c9266838b169
COVID-2502.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2502.png,COVID/images/COVID-2502.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2502.png,COVID/masks/COVID-2502.png,True,False,299,299,L,110.65,8a936dbcb483e52c5cff9c82eafa11dc
COVID-2503.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2503.png,COVID/images/COVID-2503.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2503.png,COVID/masks/COVID-2503.png,True,False,299,299,L,109.92,ded0bb41f73e9d5f97fca4c46ca66169
COVID-2504.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2504.png,COVID/images/COVID-2504.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2504.png,COVID/masks/COVID-2504.png,True,False,299,299,L,100.39,71da5362894f197a90ae70bec412e0c0
COVID-2505.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2505.png,COVID/images/COVID-2505.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2505.png,COVID/masks/COVID-2505.png,True,False,299,299,L,109.49,e13abb80fb2abbd6ec9df72fd18042b5
COVID-2506.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2506.png,COVID/images/COVID-2506.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2506.png,COVID/masks/COVID-2506.png,True,False,299,299,L,170.1,09ea037f77cc8b2d678574f314f7aa77
COVID-2507.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2507.png,COVID/images/COVID-2507.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2507.png,COVID/masks/COVID-2507.png,True,False,299,299,L,167.94,77930f89abceb0631fcc3f8f262831aa
COVID-2508.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2508.png,COVID/images/COVID-2508.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2508.png,COVID/masks/COVID-2508.png,True,False,299,299,L,174.84,c863436d499f4bcc4a768dae8c30ebb5
COVID-2509.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2509.png,COVID/images/COVID-2509.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2509.png,COVID/masks/COVID-2509.png,True,False,299,299,L,84.09,5cc4ce46307f390f0e33ee3c4842dcb9
COVID-251.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-251.png,COVID/images/COVID-251.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-251.png,COVID/masks/COVID-251.png,True,False,299,299,L,137.44,e837f6a296a8892fe80665586160bd3f
COVID-2510.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2510.png,COVID/images/COVID-2510.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2510.png,COVID/masks/COVID-2510.png,True,False,299,299,L,88.27,6decbbe1df7643fd0cc659495727b899
COVID-2511.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2511.png,COVID/images/COVID-2511.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2511.png,COVID/masks/COVID-2511.png,True,False,299,299,L,126.46,e2f7595fa4d62ff221304dbf5ae243fb
COVID-2512.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2512.png,COVID/images/COVID-2512.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2512.png,COVID/masks/COVID-2512.png,True,False,299,299,L,150.24,8390111c2a72ff88d46190b44b5745fd
COVID-2513.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2513.png,COVID/images/COVID-2513.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2513.png,COVID/masks/COVID-2513.png,True,False,299,299,L,146.75,2ed438d7bae9e34b026a7fb2850acb52
COVID-2514.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2514.png,COVID/images/COVID-2514.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2514.png,COVID/masks/COVID-2514.png,True,False,299,299,L,133.02,5c0211118b82e66db6996a9213f8f9d5
COVID-2515.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2515.png,COVID/images/COVID-2515.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2515.png,COVID/masks/COVID-2515.png,True,False,299,299,L,157.22,8e430040169684886df5774473609fa5
COVID-2516.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2516.png,COVID/images/COVID-2516.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2516.png,COVID/masks/COVID-2516.png,True,False,299,299,L,164.55,2a3bc3bbb1c588dababbf18e2feadf44
COVID-2517.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2517.png,COVID/images/COVID-2517.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2517.png,COVID/masks/COVID-2517.png,True,False,299,299,L,155.15,d36998a6c8048316e6585cecb6774ae9
COVID-2518.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2518.png,COVID/images/COVID-2518.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2518.png,COVID/masks/COVID-2518.png,True,False,299,299,L,141.76,7a0a1a5acd2fcb61f88d20f49c2edc5d
COVID-2519.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2519.png,COVID/images/COVID-2519.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2519.png,COVID/masks/COVID-2519.png,True,False,299,299,L,123.97,bda060585c214466a532381f86b466fb
COVID-252.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-252.png,COVID/images/COVID-252.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-252.png,COVID/masks/COVID-252.png,True,False,299,299,L,141.72,0e362bed4541ae1fb1e591c2b829d375
COVID-2520.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2520.png,COVID/images/COVID-2520.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2520.png,COVID/masks/COVID-2520.png,True,False,299,299,L,153.2,0bd97044a1ec6184b9a9c800215839a3
COVID-2521.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2521.png,COVID/images/COVID-2521.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2521.png,COVID/masks/COVID-2521.png,True,False,299,299,L,93.77,1dd0b2e2f969b91b63afaf68dd74a3f8
COVID-2522.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2522.png,COVID/images/COVID-2522.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2522.png,COVID/masks/COVID-2522.png,True,False,299,299,L,134.56,97fa568d8863149852a8293751fda399
COVID-2523.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2523.png,COVID/images/COVID-2523.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2523.png,COVID/masks/COVID-2523.png,True,False,299,299,L,161.25,374578c0e5f146006c1c317e022e878d
COVID-2524.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2524.png,COVID/images/COVID-2524.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2524.png,COVID/masks/COVID-2524.png,True,False,299,299,L,162.56,6cee250699d4f04e8971ead108da8539
COVID-2525.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2525.png,COVID/images/COVID-2525.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2525.png,COVID/masks/COVID-2525.png,True,False,299,299,L,174.21,89e733c52df64e42cbee3fb54775a703
COVID-2526.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2526.png,COVID/images/COVID-2526.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2526.png,COVID/masks/COVID-2526.png,True,False,299,299,L,136.23,3719ae76832d19ccc189622cd5566feb
COVID-2527.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2527.png,COVID/images/COVID-2527.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2527.png,COVID/masks/COVID-2527.png,True,False,299,299,L,156.72,4b4b12dd40c76dc018c9460d5a9813e1
COVID-2528.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2528.png,COVID/images/COVID-2528.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2528.png,COVID/masks/COVID-2528.png,True,False,299,299,L,163.91,f83a2d98fff8cbe100308a355df8aec1
COVID-2529.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2529.png,COVID/images/COVID-2529.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2529.png,COVID/masks/COVID-2529.png,True,False,299,299,L,143.32,368ebe34e0a29096422ee658908303fb
COVID-253.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-253.png,COVID/images/COVID-253.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-253.png,COVID/masks/COVID-253.png,True,False,299,299,L,122.87,5d695d655aef0c299ade9204e183bd1e
COVID-2530.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2530.png,COVID/images/COVID-2530.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2530.png,COVID/masks/COVID-2530.png,True,False,299,299,L,145.74,b5ecdd37d24bb54b4f11ef808542abc4
COVID-2531.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2531.png,COVID/images/COVID-2531.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2531.png,COVID/masks/COVID-2531.png,True,False,299,299,L,122.07,7316a56564ac86b2066a9991a9290507
COVID-2532.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2532.png,COVID/images/COVID-2532.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2532.png,COVID/masks/COVID-2532.png,True,False,299,299,L,137.98,0353924a8c5d63c3b487f82ebdf821e2
COVID-2533.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2533.png,COVID/images/COVID-2533.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2533.png,COVID/masks/COVID-2533.png,True,False,299,299,L,140.82,bb35bb187b18f46ca07c4e6013c45ebd
COVID-2534.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2534.png,COVID/images/COVID-2534.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2534.png,COVID/masks/COVID-2534.png,True,False,299,299,L,141.69,0519b631290bb3060f703aeb20111ab0
COVID-2535.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2535.png,COVID/images/COVID-2535.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2535.png,COVID/masks/COVID-2535.png,True,False,299,299,L,132.47,b023b769b857f58e9893e607140ba309
COVID-2536.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2536.png,COVID/images/COVID-2536.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2536.png,COVID/masks/COVID-2536.png,True,False,299,299,L,130.49,a8a2151e3dd73c8e2da047405b5b8fd9
COVID-2537.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2537.png,COVID/images/COVID-2537.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2537.png,COVID/masks/COVID-2537.png,True,False,299,299,L,144.28,2def46bb674b912ab9323d9aff1d2631
COVID-2538.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2538.png,COVID/images/COVID-2538.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2538.png,COVID/masks/COVID-2538.png,True,False,299,299,L,139.98,1aa0e17b50b43a74709cdfbbd9092878
COVID-2539.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2539.png,COVID/images/COVID-2539.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2539.png,COVID/masks/COVID-2539.png,True,False,299,299,L,140.12,81870dcfdf51c0e7f60ae5ac612096f6
COVID-254.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-254.png,COVID/images/COVID-254.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-254.png,COVID/masks/COVID-254.png,True,False,299,299,L,147.14,760fecad41a4c80abc61682b92154b52
COVID-2540.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2540.png,COVID/images/COVID-2540.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2540.png,COVID/masks/COVID-2540.png,True,False,299,299,L,178.9,26b5fd692cc3638d29232ac17abc2978
COVID-2541.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2541.png,COVID/images/COVID-2541.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2541.png,COVID/masks/COVID-2541.png,True,False,299,299,L,119.49,7083df1340d299c1f2aa9f3185a58ba4
COVID-2542.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2542.png,COVID/images/COVID-2542.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2542.png,COVID/masks/COVID-2542.png,True,False,299,299,L,76.11,c2e713ea25ba1f8ccee63c58023633d3
COVID-2543.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2543.png,COVID/images/COVID-2543.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2543.png,COVID/masks/COVID-2543.png,True,False,299,299,L,123.05,afc0532a1f717dda7983ad2b2afcfc7c
COVID-2544.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2544.png,COVID/images/COVID-2544.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2544.png,COVID/masks/COVID-2544.png,True,False,299,299,L,94.04,0fb72d5bb2b0fd5a8d8835c42fc757d4
COVID-2545.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2545.png,COVID/images/COVID-2545.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2545.png,COVID/masks/COVID-2545.png,True,False,299,299,L,117.23,58a1ecde55fd9af4274cb6e9880c514f
COVID-2546.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2546.png,COVID/images/COVID-2546.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2546.png,COVID/masks/COVID-2546.png,True,False,299,299,L,163.18,b975fb38e3f7eaf444dfc68c7c52f5c6
COVID-2547.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2547.png,COVID/images/COVID-2547.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2547.png,COVID/masks/COVID-2547.png,True,False,299,299,L,118.63,dbb9e40474aa759a70ed10fd32f7a960
COVID-2548.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2548.png,COVID/images/COVID-2548.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2548.png,COVID/masks/COVID-2548.png,True,False,299,299,L,141.56,cdbe712fc54c6be2d2df9737fca7e698
COVID-2549.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2549.png,COVID/images/COVID-2549.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2549.png,COVID/masks/COVID-2549.png,True,False,299,299,L,66.63,846dfd76c2c06819b0bc7b2b8c86cdcf
COVID-255.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-255.png,COVID/images/COVID-255.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-255.png,COVID/masks/COVID-255.png,True,False,299,299,L,141.56,91a533df29b339164bc514a0ae2ee365
COVID-2550.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2550.png,COVID/images/COVID-2550.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2550.png,COVID/masks/COVID-2550.png,True,False,299,299,L,167.76,f48af373e77f9153fbabc7c788526b61
COVID-2551.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2551.png,COVID/images/COVID-2551.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2551.png,COVID/masks/COVID-2551.png,True,False,299,299,L,71.61,b332607889f9c4e10d52fe9025716891
COVID-2552.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2552.png,COVID/images/COVID-2552.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2552.png,COVID/masks/COVID-2552.png,True,False,299,299,L,114.14,1104860a0d5c11019d116b4d76413354
COVID-2553.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2553.png,COVID/images/COVID-2553.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2553.png,COVID/masks/COVID-2553.png,True,False,299,299,L,135.45,283f4ed612002837eb7e233bbbe3cd35
COVID-2554.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2554.png,COVID/images/COVID-2554.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2554.png,COVID/masks/COVID-2554.png,True,False,299,299,L,127.31,095a91b8caba62cafb4ad20d191f7105
COVID-2555.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2555.png,COVID/images/COVID-2555.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2555.png,COVID/masks/COVID-2555.png,True,False,299,299,L,116.52,a97a42bf7e9e70a7eb542bff1a7d4696
COVID-2556.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2556.png,COVID/images/COVID-2556.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2556.png,COVID/masks/COVID-2556.png,True,False,299,299,L,130.51,3a124dd4d1be2ebea6108c3966423f0b
COVID-2557.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2557.png,COVID/images/COVID-2557.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2557.png,COVID/masks/COVID-2557.png,True,False,299,299,L,78.58,4f004626eefa5819a18632861bbab510
COVID-2558.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2558.png,COVID/images/COVID-2558.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2558.png,COVID/masks/COVID-2558.png,True,False,299,299,L,132.6,533814509d91da5a5e123e57514c0301
COVID-2559.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2559.png,COVID/images/COVID-2559.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2559.png,COVID/masks/COVID-2559.png,True,False,299,299,L,140.51,abae94d7222588f9877c38ab6cfe38a3
COVID-256.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-256.png,COVID/images/COVID-256.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-256.png,COVID/masks/COVID-256.png,True,False,299,299,L,152.14,0a07d8ac5e58bc1c644f84ba7717919c
COVID-2560.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2560.png,COVID/images/COVID-2560.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2560.png,COVID/masks/COVID-2560.png,True,False,299,299,L,168.83,968ae0fa3ece72d6ff6bbd7c6e0b533a
COVID-2561.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2561.png,COVID/images/COVID-2561.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2561.png,COVID/masks/COVID-2561.png,True,False,299,299,L,131.65,751c6d0ddc40108aec6eab5bdb515493
COVID-2562.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2562.png,COVID/images/COVID-2562.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2562.png,COVID/masks/COVID-2562.png,True,False,299,299,L,152.8,8fbcc2627023fbe4363d6ecc4dc551f9
COVID-2563.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2563.png,COVID/images/COVID-2563.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2563.png,COVID/masks/COVID-2563.png,True,False,299,299,L,132.36,a9ea3f90f889076bbed51c42c302f5eb
COVID-2564.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2564.png,COVID/images/COVID-2564.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2564.png,COVID/masks/COVID-2564.png,True,False,299,299,L,123.76,eee56fd57d1a348e6fa20fce6ab74238
COVID-2565.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2565.png,COVID/images/COVID-2565.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2565.png,COVID/masks/COVID-2565.png,True,False,299,299,L,133.37,33f977b01df4f473a8ebd038ab12ac68
COVID-2566.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2566.png,COVID/images/COVID-2566.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2566.png,COVID/masks/COVID-2566.png,True,False,299,299,L,129.38,dd578f21da13abb1aea542a41d9af1ff
COVID-2567.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2567.png,COVID/images/COVID-2567.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2567.png,COVID/masks/COVID-2567.png,True,False,299,299,L,158.09,59cf3fbbbec0d4920dd160e8b405f918
COVID-2568.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2568.png,COVID/images/COVID-2568.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2568.png,COVID/masks/COVID-2568.png,True,False,299,299,L,100.94,3ed6315203eeeaf59c83b92b1b2abb38
COVID-2569.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2569.png,COVID/images/COVID-2569.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2569.png,COVID/masks/COVID-2569.png,True,False,299,299,L,143.46,e23a4f69bdafc3acc536b7528ba7ca3a
COVID-257.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-257.png,COVID/images/COVID-257.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-257.png,COVID/masks/COVID-257.png,True,False,299,299,L,131.95,cfea57004eb4a25f750d5460e8d274e5
COVID-2570.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2570.png,COVID/images/COVID-2570.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2570.png,COVID/masks/COVID-2570.png,True,False,299,299,L,140.87,4498c68a1400dd4803e9021753c2b8b4
COVID-2571.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2571.png,COVID/images/COVID-2571.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2571.png,COVID/masks/COVID-2571.png,True,False,299,299,L,150.06,885418c0c04ea63ecc7a84093ae6eb88
COVID-2572.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2572.png,COVID/images/COVID-2572.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2572.png,COVID/masks/COVID-2572.png,True,False,299,299,L,131.94,4ee21ee2a2afe5b31b7b72a646f4f7db
COVID-2573.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2573.png,COVID/images/COVID-2573.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2573.png,COVID/masks/COVID-2573.png,True,False,299,299,L,132.34,0f0a89b9b2e53fd1b9eb6091f048d2f1
COVID-2574.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2574.png,COVID/images/COVID-2574.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2574.png,COVID/masks/COVID-2574.png,True,False,299,299,L,110.76,a58eccd44ab7b45d89479e5240767380
COVID-2575.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2575.png,COVID/images/COVID-2575.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2575.png,COVID/masks/COVID-2575.png,True,False,299,299,L,111.19,d12246e4a493c26458e8c21c3369cf2e
COVID-2576.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2576.png,COVID/images/COVID-2576.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2576.png,COVID/masks/COVID-2576.png,True,False,299,299,L,84.52,fa992a8c28c04f1dd956b7fc781a145c
COVID-2577.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2577.png,COVID/images/COVID-2577.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2577.png,COVID/masks/COVID-2577.png,True,False,299,299,L,82.45,2a9f940c67a66a201ad203eaa417324e
COVID-2578.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2578.png,COVID/images/COVID-2578.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2578.png,COVID/masks/COVID-2578.png,True,False,299,299,L,75.37,ceabf7d50e4a99fd758a109719b1a087
COVID-2579.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2579.png,COVID/images/COVID-2579.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2579.png,COVID/masks/COVID-2579.png,True,False,299,299,L,70.55,1408b1980239346638d8f55114c501de
COVID-258.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-258.png,COVID/images/COVID-258.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-258.png,COVID/masks/COVID-258.png,True,False,299,299,L,129.42,fb7b5c9e74844742c5598bff05a4ca21
COVID-2580.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2580.png,COVID/images/COVID-2580.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2580.png,COVID/masks/COVID-2580.png,True,False,299,299,L,77.98,86b49898d42b28d66006198607848676
COVID-2581.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2581.png,COVID/images/COVID-2581.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2581.png,COVID/masks/COVID-2581.png,True,False,299,299,L,119.24,6896b9a3517a54325663181ad98d9fbf
COVID-2582.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2582.png,COVID/images/COVID-2582.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2582.png,COVID/masks/COVID-2582.png,True,False,299,299,L,144.56,102036fefa9b98b23bbfb0ff2aa6acd0
COVID-2583.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2583.png,COVID/images/COVID-2583.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2583.png,COVID/masks/COVID-2583.png,True,False,299,299,L,143.18,97dd650e34b516ed85f6099820d26817
COVID-2584.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2584.png,COVID/images/COVID-2584.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2584.png,COVID/masks/COVID-2584.png,True,False,299,299,L,148.59,d3dfcc76d5236b69e2fa6ceb3f8ad4e3
COVID-2585.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2585.png,COVID/images/COVID-2585.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2585.png,COVID/masks/COVID-2585.png,True,False,299,299,L,133.42,b5ddae86da6565a5efc7bbfc9981f669
COVID-2586.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2586.png,COVID/images/COVID-2586.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2586.png,COVID/masks/COVID-2586.png,True,False,299,299,L,137.76,a53ab128a5f69324cff1244053604096
COVID-2587.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2587.png,COVID/images/COVID-2587.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2587.png,COVID/masks/COVID-2587.png,True,False,299,299,L,145.58,26361992be100572313176de34c49e96
COVID-2588.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2588.png,COVID/images/COVID-2588.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2588.png,COVID/masks/COVID-2588.png,True,False,299,299,L,138.76,dec9f3262862bac09322e1e48f3e46af
COVID-2589.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2589.png,COVID/images/COVID-2589.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2589.png,COVID/masks/COVID-2589.png,True,False,299,299,L,133.44,03254a433d244ed073d77c9a3187dceb
COVID-259.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-259.png,COVID/images/COVID-259.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-259.png,COVID/masks/COVID-259.png,True,False,299,299,L,134.6,2b62940f0b602d63aa0f49153fa50199
COVID-2590.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2590.png,COVID/images/COVID-2590.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2590.png,COVID/masks/COVID-2590.png,True,False,299,299,L,159.05,e5420a67612f970dbd87f20ead8dd2df
COVID-2591.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2591.png,COVID/images/COVID-2591.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2591.png,COVID/masks/COVID-2591.png,True,False,299,299,L,139.21,2ee04717172a55e77e1a37a2c1b72f64
COVID-2592.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2592.png,COVID/images/COVID-2592.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2592.png,COVID/masks/COVID-2592.png,True,False,299,299,L,134.95,019b1cede7c87e72426863b7062e9cfa
COVID-2593.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2593.png,COVID/images/COVID-2593.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2593.png,COVID/masks/COVID-2593.png,True,False,299,299,L,161.12,b9ec9bef22442c61f513ff3aec6614e3
COVID-2594.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2594.png,COVID/images/COVID-2594.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2594.png,COVID/masks/COVID-2594.png,True,False,299,299,L,136.3,8d6b4d0e5eebf116840ec16b0cffc76e
COVID-2595.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2595.png,COVID/images/COVID-2595.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2595.png,COVID/masks/COVID-2595.png,True,False,299,299,L,114.98,16e0c4bdd943e95c271b6ac0a6f7407c
COVID-2596.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2596.png,COVID/images/COVID-2596.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2596.png,COVID/masks/COVID-2596.png,True,False,299,299,L,95.3,dbf7e9fba901986f50cd5256646da990
COVID-2597.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2597.png,COVID/images/COVID-2597.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2597.png,COVID/masks/COVID-2597.png,True,False,299,299,L,91.56,35bdcec9b73162af47bee6154e437576
COVID-2598.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2598.png,COVID/images/COVID-2598.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2598.png,COVID/masks/COVID-2598.png,True,False,299,299,L,179.55,9f2d9f7f13fc304f018e2829c81c8be9
COVID-2599.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2599.png,COVID/images/COVID-2599.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2599.png,COVID/masks/COVID-2599.png,True,False,299,299,L,134.81,0bd7e88a75587c5757adacf3991fbb82
COVID-26.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-26.png,COVID/images/COVID-26.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-26.png,COVID/masks/COVID-26.png,True,False,299,299,L,111.75,89a1a45e13a098ce3e23aa573d9c73bd
COVID-260.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-260.png,COVID/images/COVID-260.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-260.png,COVID/masks/COVID-260.png,True,False,299,299,L,123.11,51b1582e2ce374d9be8eefcdf0548bc1
COVID-2600.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2600.png,COVID/images/COVID-2600.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2600.png,COVID/masks/COVID-2600.png,True,False,299,299,L,151.75,0b204f45b798773417e7e164268a646d
COVID-2601.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2601.png,COVID/images/COVID-2601.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2601.png,COVID/masks/COVID-2601.png,True,False,299,299,L,107.4,b38fcb546773330fda235f91ad5bc76f
COVID-2602.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2602.png,COVID/images/COVID-2602.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2602.png,COVID/masks/COVID-2602.png,True,False,299,299,L,137.18,78067e0b74e599b8f704e05c71b71e26
COVID-2603.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2603.png,COVID/images/COVID-2603.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2603.png,COVID/masks/COVID-2603.png,True,False,299,299,L,131.75,7948285f5eb04bb5bf69e12d0b2f5d5c
COVID-2604.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2604.png,COVID/images/COVID-2604.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2604.png,COVID/masks/COVID-2604.png,True,False,299,299,L,138.94,0a7cee93b663d75b2c6e239b99558f33
COVID-2605.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2605.png,COVID/images/COVID-2605.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2605.png,COVID/masks/COVID-2605.png,True,False,299,299,L,147.14,fba7596edd43ce0a3a0324851010a248
COVID-2606.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2606.png,COVID/images/COVID-2606.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2606.png,COVID/masks/COVID-2606.png,True,False,299,299,L,176.69,529b658aed22be7c92826ad743a1bec7
COVID-2607.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2607.png,COVID/images/COVID-2607.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2607.png,COVID/masks/COVID-2607.png,True,False,299,299,L,164.97,d7029c3af0f66630e4e7aed5d5cac2bf
COVID-2608.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2608.png,COVID/images/COVID-2608.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2608.png,COVID/masks/COVID-2608.png,True,False,299,299,L,160.59,619db41ad3adc4c61b7a2958b4792d8e
COVID-2609.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2609.png,COVID/images/COVID-2609.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2609.png,COVID/masks/COVID-2609.png,True,False,299,299,L,147.76,2c1043ce3fc84212a747d923aab6440d
COVID-261.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-261.png,COVID/images/COVID-261.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-261.png,COVID/masks/COVID-261.png,True,False,299,299,L,143.24,624c5f3bfb947009e2028a66e2a86ad8
COVID-2610.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2610.png,COVID/images/COVID-2610.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2610.png,COVID/masks/COVID-2610.png,True,False,299,299,L,172.76,01019916c069a4e6056146c91d9568e4
COVID-2611.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2611.png,COVID/images/COVID-2611.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2611.png,COVID/masks/COVID-2611.png,True,False,299,299,L,145.76,5ad6bc578831be54bb8099a7d2a5c037
COVID-2612.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2612.png,COVID/images/COVID-2612.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2612.png,COVID/masks/COVID-2612.png,True,False,299,299,L,145.2,24dd00b55484885173d034ceaa49446c
COVID-2613.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2613.png,COVID/images/COVID-2613.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2613.png,COVID/masks/COVID-2613.png,True,False,299,299,L,132.58,04b1c83fa6f79db846bf3924932410d9
COVID-2614.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2614.png,COVID/images/COVID-2614.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2614.png,COVID/masks/COVID-2614.png,True,False,299,299,L,118.87,16c237234d647c7226fde4da9bffc32c
COVID-2615.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2615.png,COVID/images/COVID-2615.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2615.png,COVID/masks/COVID-2615.png,True,False,299,299,L,139.1,58d157b4d91ab18259aed10dce7ed8d4
COVID-2616.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2616.png,COVID/images/COVID-2616.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2616.png,COVID/masks/COVID-2616.png,True,False,299,299,L,137.1,0e47776fc641c3afdd96f531886ea912
COVID-2617.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2617.png,COVID/images/COVID-2617.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2617.png,COVID/masks/COVID-2617.png,True,False,299,299,L,142.55,b915de41516604e2ef1b61221ecab067
COVID-2618.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2618.png,COVID/images/COVID-2618.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2618.png,COVID/masks/COVID-2618.png,True,False,299,299,L,120.27,8d1b4062cc1e6152c51fec99d1d12d0b
COVID-2619.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2619.png,COVID/images/COVID-2619.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2619.png,COVID/masks/COVID-2619.png,True,False,299,299,L,165.41,555eb93e4f2383c3ea9d13a6fb155cee
COVID-262.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-262.png,COVID/images/COVID-262.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-262.png,COVID/masks/COVID-262.png,True,False,299,299,L,151.71,1bb9fa6c6262f64ed0b1137e66fd706a
COVID-2620.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2620.png,COVID/images/COVID-2620.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2620.png,COVID/masks/COVID-2620.png,True,False,299,299,L,103.06,c806bbf9bdb17ba092338c59e1c3410d
COVID-2621.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2621.png,COVID/images/COVID-2621.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2621.png,COVID/masks/COVID-2621.png,True,False,299,299,L,186.23,08db2649a7f01435d2c8d993fabb5026
COVID-2622.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2622.png,COVID/images/COVID-2622.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2622.png,COVID/masks/COVID-2622.png,True,False,299,299,L,147.66,68789b418347e1dda3241efa8ab49d10
COVID-2623.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2623.png,COVID/images/COVID-2623.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2623.png,COVID/masks/COVID-2623.png,True,False,299,299,L,156.88,756e6cc16b7f08c4a8debd3acd6bb47f
COVID-2624.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2624.png,COVID/images/COVID-2624.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2624.png,COVID/masks/COVID-2624.png,True,False,299,299,L,147.02,c04478f490abc30be3c626ed304ee1e1
COVID-2625.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2625.png,COVID/images/COVID-2625.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2625.png,COVID/masks/COVID-2625.png,True,False,299,299,L,120.11,021baf6fdab2b601d4ee1991d7ba19a7
COVID-2626.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2626.png,COVID/images/COVID-2626.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2626.png,COVID/masks/COVID-2626.png,True,False,299,299,L,115.72,279dfb71eeca0b5620ed8ee896d13461
COVID-2627.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2627.png,COVID/images/COVID-2627.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2627.png,COVID/masks/COVID-2627.png,True,False,299,299,L,142.28,4ced73b24faec0528b89c7fbb627191c
COVID-2628.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2628.png,COVID/images/COVID-2628.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2628.png,COVID/masks/COVID-2628.png,True,False,299,299,L,83.38,3ccf56e0389b973e4fbc884131757be6
COVID-2629.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2629.png,COVID/images/COVID-2629.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2629.png,COVID/masks/COVID-2629.png,True,False,299,299,L,153.01,223b7070067a528d474fe7a7f70eb947
COVID-263.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-263.png,COVID/images/COVID-263.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-263.png,COVID/masks/COVID-263.png,True,False,299,299,L,110.44,a367a35648fbc180d21f95883abf6dfb
COVID-2630.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2630.png,COVID/images/COVID-2630.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2630.png,COVID/masks/COVID-2630.png,True,False,299,299,L,162.29,5ca6cc3ac34e5a45c359ea23f0e84e99
COVID-2631.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2631.png,COVID/images/COVID-2631.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2631.png,COVID/masks/COVID-2631.png,True,False,299,299,L,86.12,483c6e2733a6ecae9f9d9dce7fa3a557
COVID-2632.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2632.png,COVID/images/COVID-2632.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2632.png,COVID/masks/COVID-2632.png,True,False,299,299,L,162.92,878dce327a7e48b2f63f6b72bec6fa25
COVID-2633.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2633.png,COVID/images/COVID-2633.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2633.png,COVID/masks/COVID-2633.png,True,False,299,299,L,153.11,bf1ca50bd35c917666f0a0544601ffe3
COVID-2634.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2634.png,COVID/images/COVID-2634.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2634.png,COVID/masks/COVID-2634.png,True,False,299,299,L,168.63,d9d1a180ab8f0db5d00e541e576e410d
COVID-2635.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2635.png,COVID/images/COVID-2635.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2635.png,COVID/masks/COVID-2635.png,True,False,299,299,L,143.79,ab2a54b5dcfd5259fbba64d4280c0bed
COVID-2636.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2636.png,COVID/images/COVID-2636.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2636.png,COVID/masks/COVID-2636.png,True,False,299,299,L,111.3,13cc27cf58a7b1971f87b4fc4c20248c
COVID-2637.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2637.png,COVID/images/COVID-2637.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2637.png,COVID/masks/COVID-2637.png,True,False,299,299,L,118.03,92714a4444f375beced28d3ef6f636d0
COVID-2638.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2638.png,COVID/images/COVID-2638.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2638.png,COVID/masks/COVID-2638.png,True,False,299,299,L,128.35,050c0eda6117ef26f671ec6cd5b07113
COVID-2639.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2639.png,COVID/images/COVID-2639.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2639.png,COVID/masks/COVID-2639.png,True,False,299,299,L,138.83,f07b4a8e731f7f802d16eebfdb1040be
COVID-264.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-264.png,COVID/images/COVID-264.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-264.png,COVID/masks/COVID-264.png,True,False,299,299,L,171.72,88d193e3999aa1e56445d18e6a8dd7a5
COVID-2640.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2640.png,COVID/images/COVID-2640.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2640.png,COVID/masks/COVID-2640.png,True,False,299,299,L,152.14,3353825bf28ec48885f1f8de5ad554ef
COVID-2641.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2641.png,COVID/images/COVID-2641.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2641.png,COVID/masks/COVID-2641.png,True,False,299,299,L,125.37,25e85213424a429456e718ccd7a0c88a
COVID-2642.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2642.png,COVID/images/COVID-2642.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2642.png,COVID/masks/COVID-2642.png,True,False,299,299,L,138.28,71e251624af2c995e55a6636da2fbb5b
COVID-2643.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2643.png,COVID/images/COVID-2643.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2643.png,COVID/masks/COVID-2643.png,True,False,299,299,L,170.31,c8ead6dd4bce3d65a4448fee4a16e6af
COVID-2644.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2644.png,COVID/images/COVID-2644.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2644.png,COVID/masks/COVID-2644.png,True,False,299,299,L,138.17,d8c088b9fe733784cb847bea7d879889
COVID-2645.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2645.png,COVID/images/COVID-2645.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2645.png,COVID/masks/COVID-2645.png,True,False,299,299,L,159.28,1be85d8ae136099047096801cfba8560
COVID-2646.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2646.png,COVID/images/COVID-2646.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2646.png,COVID/masks/COVID-2646.png,True,False,299,299,L,178.54,bffb4f676ed585580d72d34f4ae4a4ca
COVID-2647.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2647.png,COVID/images/COVID-2647.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2647.png,COVID/masks/COVID-2647.png,True,False,299,299,L,165.55,4d33596984c96c5790a614b833aafde6
COVID-2648.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2648.png,COVID/images/COVID-2648.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2648.png,COVID/masks/COVID-2648.png,True,False,299,299,L,161.09,e408d443e6d69dadc3c1ccf533926b4b
COVID-2649.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2649.png,COVID/images/COVID-2649.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2649.png,COVID/masks/COVID-2649.png,True,False,299,299,L,129.01,eeca9d9c1954890c7296607d39391b8f
COVID-265.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-265.png,COVID/images/COVID-265.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-265.png,COVID/masks/COVID-265.png,True,False,299,299,L,123.08,1662173a7b6d276601569097b41ed169
COVID-2650.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2650.png,COVID/images/COVID-2650.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2650.png,COVID/masks/COVID-2650.png,True,False,299,299,L,138.17,22f1cb4fcda642e442f1cdc004bc5e04
COVID-2651.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2651.png,COVID/images/COVID-2651.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2651.png,COVID/masks/COVID-2651.png,True,False,299,299,L,97.38,0ad25221c746216c32cbdb962bae5ad3
COVID-2652.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2652.png,COVID/images/COVID-2652.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2652.png,COVID/masks/COVID-2652.png,True,False,299,299,L,167.41,15b134e3aa246472c5b109f844cd2c98
COVID-2653.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2653.png,COVID/images/COVID-2653.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2653.png,COVID/masks/COVID-2653.png,True,False,299,299,L,135.78,ec41cbf334ad15947a02a9460c1be252
COVID-2654.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2654.png,COVID/images/COVID-2654.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2654.png,COVID/masks/COVID-2654.png,True,False,299,299,L,133.39,d6ad2a2551b3e9f75d5d1696646ff329
COVID-2655.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2655.png,COVID/images/COVID-2655.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2655.png,COVID/masks/COVID-2655.png,True,False,299,299,L,146.27,8df951a20a19f1862e7134ff79ad2082
COVID-2656.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2656.png,COVID/images/COVID-2656.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2656.png,COVID/masks/COVID-2656.png,True,False,299,299,L,159.19,7d4a1592f4b508828feaf04637bf1a10
COVID-2657.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2657.png,COVID/images/COVID-2657.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2657.png,COVID/masks/COVID-2657.png,True,False,299,299,L,165.78,4921d11114c714a4f0cb338dccecd737
COVID-2658.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2658.png,COVID/images/COVID-2658.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2658.png,COVID/masks/COVID-2658.png,True,False,299,299,L,139.11,418d426c1e50d2ca1c0699d2fbdd77d1
COVID-2659.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2659.png,COVID/images/COVID-2659.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2659.png,COVID/masks/COVID-2659.png,True,False,299,299,L,99.95,bcadf9b80ba1d375c76851d0d4d7c62a
COVID-266.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-266.png,COVID/images/COVID-266.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-266.png,COVID/masks/COVID-266.png,True,False,299,299,L,159.2,3f88e85a14feb1826475be2ef0a7aeb9
COVID-2660.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2660.png,COVID/images/COVID-2660.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2660.png,COVID/masks/COVID-2660.png,True,False,299,299,L,130.89,60614f6ca31ca928fffb09f88f514d77
COVID-2661.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2661.png,COVID/images/COVID-2661.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2661.png,COVID/masks/COVID-2661.png,True,False,299,299,L,91.42,c6d1f0af297860828f44464723c2fe79
COVID-2662.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2662.png,COVID/images/COVID-2662.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2662.png,COVID/masks/COVID-2662.png,True,False,299,299,L,146.04,0c2c2526e356d583540d1797bb759a5d
COVID-2663.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2663.png,COVID/images/COVID-2663.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2663.png,COVID/masks/COVID-2663.png,True,False,299,299,L,127.67,c1b44f193125b18a19b8d31841f5a7dc
COVID-2664.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2664.png,COVID/images/COVID-2664.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2664.png,COVID/masks/COVID-2664.png,True,False,299,299,L,135.78,2090411e078f7f4864bf62bdce17f7c3
COVID-2665.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2665.png,COVID/images/COVID-2665.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2665.png,COVID/masks/COVID-2665.png,True,False,299,299,L,129.82,dad2af8711cf47b665e94824efb6dd5e
COVID-2666.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2666.png,COVID/images/COVID-2666.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2666.png,COVID/masks/COVID-2666.png,True,False,299,299,L,103.82,3e9cd7fd6af03cb9f0760f3f5452240b
COVID-2667.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2667.png,COVID/images/COVID-2667.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2667.png,COVID/masks/COVID-2667.png,True,False,299,299,L,103.82,3e9cd7fd6af03cb9f0760f3f5452240b
COVID-2668.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2668.png,COVID/images/COVID-2668.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2668.png,COVID/masks/COVID-2668.png,True,False,299,299,L,160.66,124130654115aaf4c4518c25de75ccbb
COVID-2669.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2669.png,COVID/images/COVID-2669.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2669.png,COVID/masks/COVID-2669.png,True,False,299,299,L,130.84,6c948e5bcd44f719221f216aa566579d
COVID-267.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-267.png,COVID/images/COVID-267.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-267.png,COVID/masks/COVID-267.png,True,False,299,299,L,149.4,9ffb5c65984a2282c58e83e577c3148e
COVID-2670.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2670.png,COVID/images/COVID-2670.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2670.png,COVID/masks/COVID-2670.png,True,False,299,299,L,140.54,04e683f7923643d4a4fd3695e5d993aa
COVID-2671.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2671.png,COVID/images/COVID-2671.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2671.png,COVID/masks/COVID-2671.png,True,False,299,299,L,137.06,7a41038b3b51da84c5775f06005d0ddb
COVID-2672.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2672.png,COVID/images/COVID-2672.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2672.png,COVID/masks/COVID-2672.png,True,False,299,299,L,154.25,e20f2d3c27564418a11a7108818b55d3
COVID-2673.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2673.png,COVID/images/COVID-2673.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2673.png,COVID/masks/COVID-2673.png,True,False,299,299,L,137.78,690abe1a88f8747a8adc7cfe7d2995e8
COVID-2674.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2674.png,COVID/images/COVID-2674.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2674.png,COVID/masks/COVID-2674.png,True,False,299,299,L,111.85,f30711d0160c6e3f2bda303106e332c1
COVID-2675.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2675.png,COVID/images/COVID-2675.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2675.png,COVID/masks/COVID-2675.png,True,False,299,299,L,81.38,e7611f3de6c179d3ec0fa2d5c3b28c09
COVID-2676.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2676.png,COVID/images/COVID-2676.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2676.png,COVID/masks/COVID-2676.png,True,False,299,299,L,129.74,82f2f2c359f2f11a17d040e47e0a1ac9
COVID-2677.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2677.png,COVID/images/COVID-2677.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2677.png,COVID/masks/COVID-2677.png,True,False,299,299,L,134.93,eacc92e9c5c3dcddea6daef8ddf76837
COVID-2678.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2678.png,COVID/images/COVID-2678.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2678.png,COVID/masks/COVID-2678.png,True,False,299,299,L,134.93,eacc92e9c5c3dcddea6daef8ddf76837
COVID-2679.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2679.png,COVID/images/COVID-2679.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2679.png,COVID/masks/COVID-2679.png,True,False,299,299,L,76.51,266452b6feb51542a8a9ea7cc0fe4e24
COVID-268.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-268.png,COVID/images/COVID-268.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-268.png,COVID/masks/COVID-268.png,True,False,299,299,L,157.36,32c7fd4defc7f6f8cf9c03baf9a8bdc3
COVID-2680.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2680.png,COVID/images/COVID-2680.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2680.png,COVID/masks/COVID-2680.png,True,False,299,299,L,153.09,a2a46f31e8c816a39b9fa4e98c992f93
COVID-2681.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2681.png,COVID/images/COVID-2681.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2681.png,COVID/masks/COVID-2681.png,True,False,299,299,L,143.42,3d74749089274c55f9ef70d1c73df43e
COVID-2682.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2682.png,COVID/images/COVID-2682.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2682.png,COVID/masks/COVID-2682.png,True,False,299,299,L,150.76,16eeb28cf3af938968603b7d2fe2d6a8
COVID-2683.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2683.png,COVID/images/COVID-2683.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2683.png,COVID/masks/COVID-2683.png,True,False,299,299,L,135.57,0755ad7a56b2b2295f362a94bdb49795
COVID-2684.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2684.png,COVID/images/COVID-2684.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2684.png,COVID/masks/COVID-2684.png,True,False,299,299,L,159.1,d38a5d228c8521db9121f8194cb0d476
COVID-2685.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2685.png,COVID/images/COVID-2685.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2685.png,COVID/masks/COVID-2685.png,True,False,299,299,L,185.91,41005f9f4adb5dcb7a16fcdf44a94724
COVID-2686.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2686.png,COVID/images/COVID-2686.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2686.png,COVID/masks/COVID-2686.png,True,False,299,299,L,122.21,7820fa931a45329cacdc9d58c0aa2b4e
COVID-2687.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2687.png,COVID/images/COVID-2687.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2687.png,COVID/masks/COVID-2687.png,True,False,299,299,L,182.1,d6216b03fb1a630417505ffdf0f13191
COVID-2688.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2688.png,COVID/images/COVID-2688.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2688.png,COVID/masks/COVID-2688.png,True,False,299,299,L,161.45,62cc337419b80d5b83f93ede0eadd530
COVID-2689.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2689.png,COVID/images/COVID-2689.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2689.png,COVID/masks/COVID-2689.png,True,False,299,299,L,162.74,96867f8cdfcb715a01a81429c7d94e5f
COVID-269.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-269.png,COVID/images/COVID-269.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-269.png,COVID/masks/COVID-269.png,True,False,299,299,L,131.35,f46de0bf856a82f721c9c78f6c25325c
COVID-2690.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2690.png,COVID/images/COVID-2690.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2690.png,COVID/masks/COVID-2690.png,True,False,299,299,L,79.21,93ef6a6e887deca9bc9ed43d79daef57
COVID-2691.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2691.png,COVID/images/COVID-2691.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2691.png,COVID/masks/COVID-2691.png,True,False,299,299,L,129.68,1c7a1fba266c21c07765dc07c2a9966f
COVID-2692.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2692.png,COVID/images/COVID-2692.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2692.png,COVID/masks/COVID-2692.png,True,False,299,299,L,109.9,b317cf264f9daa631a10f7381ae692bb
COVID-2693.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2693.png,COVID/images/COVID-2693.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2693.png,COVID/masks/COVID-2693.png,True,False,299,299,L,138.11,b4a7cde7562d185be66505ba8212bc3f
COVID-2694.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2694.png,COVID/images/COVID-2694.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2694.png,COVID/masks/COVID-2694.png,True,False,299,299,L,112.81,dbebbf1d0a3d542daac02617850101b3
COVID-2695.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2695.png,COVID/images/COVID-2695.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2695.png,COVID/masks/COVID-2695.png,True,False,299,299,L,169.12,397040378bf7165e973334e7f73684f8
COVID-2696.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2696.png,COVID/images/COVID-2696.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2696.png,COVID/masks/COVID-2696.png,True,False,299,299,L,116.3,d6f9262460bebc9b1aefb82fac93dc19
COVID-2697.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2697.png,COVID/images/COVID-2697.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2697.png,COVID/masks/COVID-2697.png,True,False,299,299,L,166.54,9ba30da09e542f7a0a7e218e547ee2bb
COVID-2698.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2698.png,COVID/images/COVID-2698.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2698.png,COVID/masks/COVID-2698.png,True,False,299,299,L,161.46,093a171ab3905296fbc4d68dec88783d
COVID-2699.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2699.png,COVID/images/COVID-2699.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2699.png,COVID/masks/COVID-2699.png,True,False,299,299,L,111.06,4d2e1ed3d084098bc8ed73febd8be1e6
COVID-27.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-27.png,COVID/images/COVID-27.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-27.png,COVID/masks/COVID-27.png,True,False,299,299,L,134.73,97664faf273e0cc5da6d9f5b4801e199
COVID-270.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-270.png,COVID/images/COVID-270.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-270.png,COVID/masks/COVID-270.png,True,False,299,299,L,180.46,3c87443502a630e9a5c15a3a22f39724
COVID-2700.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2700.png,COVID/images/COVID-2700.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2700.png,COVID/masks/COVID-2700.png,True,False,299,299,L,179.03,87cd56a40ca1b528fc5a704a9ab20c90
COVID-2701.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2701.png,COVID/images/COVID-2701.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2701.png,COVID/masks/COVID-2701.png,True,False,299,299,L,177.73,cd9d1c9d6d11ef7b17b411e65fbca5e0
COVID-2702.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2702.png,COVID/images/COVID-2702.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2702.png,COVID/masks/COVID-2702.png,True,False,299,299,L,154.86,f4ce424783783263e69656efcb94f5ca
COVID-2703.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2703.png,COVID/images/COVID-2703.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2703.png,COVID/masks/COVID-2703.png,True,False,299,299,L,154.54,ed4bd983b2751805e913edd060e81d4b
COVID-2704.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2704.png,COVID/images/COVID-2704.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2704.png,COVID/masks/COVID-2704.png,True,False,299,299,L,143.29,bcebdc5473d8483385eadbfbafa4325f
COVID-2705.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2705.png,COVID/images/COVID-2705.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2705.png,COVID/masks/COVID-2705.png,True,False,299,299,L,132.2,b7a057b4192ae5b1fd3354b35f757bd6
COVID-2706.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2706.png,COVID/images/COVID-2706.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2706.png,COVID/masks/COVID-2706.png,True,False,299,299,L,174.08,7cbf769eca7ec4f8fdadb22a33e2631c
COVID-2707.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2707.png,COVID/images/COVID-2707.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2707.png,COVID/masks/COVID-2707.png,True,False,299,299,L,126.76,39ca146291322e9a1decb57d308dde2d
COVID-2708.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2708.png,COVID/images/COVID-2708.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2708.png,COVID/masks/COVID-2708.png,True,False,299,299,L,133.43,534dd64dc5e56610ac8585e40d51b27f
COVID-2709.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2709.png,COVID/images/COVID-2709.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2709.png,COVID/masks/COVID-2709.png,True,False,299,299,L,158.85,5f97c451763666e93934a1374aa28079
COVID-271.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-271.png,COVID/images/COVID-271.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-271.png,COVID/masks/COVID-271.png,True,False,299,299,L,187.63,659dca501b27549850cdca0fb34ee164
COVID-2710.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2710.png,COVID/images/COVID-2710.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2710.png,COVID/masks/COVID-2710.png,True,False,299,299,L,98.61,cd70d93ed66e8b6e8f27f0211fbe1cd7
COVID-2711.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2711.png,COVID/images/COVID-2711.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2711.png,COVID/masks/COVID-2711.png,True,False,299,299,L,154.79,6941b4bb7f65124c50dc656ba233d600
COVID-2712.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2712.png,COVID/images/COVID-2712.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2712.png,COVID/masks/COVID-2712.png,True,False,299,299,L,110.37,05857c9a10e971f75362411527998902
COVID-2713.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2713.png,COVID/images/COVID-2713.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2713.png,COVID/masks/COVID-2713.png,True,False,299,299,L,134.98,d019c1905145cc2ea9f5eee79c3c04d6
COVID-2714.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2714.png,COVID/images/COVID-2714.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2714.png,COVID/masks/COVID-2714.png,True,False,299,299,L,175.62,d9220dac5068ac10b38dc03d3700231e
COVID-2715.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2715.png,COVID/images/COVID-2715.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2715.png,COVID/masks/COVID-2715.png,True,False,299,299,L,130.01,e27ad0c30ae29e488fa6e57ff63a4a1e
COVID-2716.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2716.png,COVID/images/COVID-2716.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2716.png,COVID/masks/COVID-2716.png,True,False,299,299,L,130.01,e27ad0c30ae29e488fa6e57ff63a4a1e
COVID-2717.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2717.png,COVID/images/COVID-2717.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2717.png,COVID/masks/COVID-2717.png,True,False,299,299,L,130.01,e27ad0c30ae29e488fa6e57ff63a4a1e
COVID-2718.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2718.png,COVID/images/COVID-2718.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2718.png,COVID/masks/COVID-2718.png,True,False,299,299,L,130.01,e27ad0c30ae29e488fa6e57ff63a4a1e
COVID-2719.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2719.png,COVID/images/COVID-2719.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2719.png,COVID/masks/COVID-2719.png,True,False,299,299,L,130.01,e27ad0c30ae29e488fa6e57ff63a4a1e
COVID-272.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-272.png,COVID/images/COVID-272.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-272.png,COVID/masks/COVID-272.png,True,False,299,299,L,145.51,cb780a9e10ba9a2c4bd9467aadc11954
COVID-2720.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2720.png,COVID/images/COVID-2720.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2720.png,COVID/masks/COVID-2720.png,True,False,299,299,L,175.28,ca3d199cfbb46f0914c5d872ffb94fbd
COVID-2721.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2721.png,COVID/images/COVID-2721.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2721.png,COVID/masks/COVID-2721.png,True,False,299,299,L,135.78,38b7ce2230cbd2f31a0868126e406767
COVID-2722.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2722.png,COVID/images/COVID-2722.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2722.png,COVID/masks/COVID-2722.png,True,False,299,299,L,116.68,fe4939d6ebc7f52232e7c04af729f64a
COVID-2723.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2723.png,COVID/images/COVID-2723.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2723.png,COVID/masks/COVID-2723.png,True,False,299,299,L,162.56,cc45b38d3d7a285fe973c28732e80dc2
COVID-2724.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2724.png,COVID/images/COVID-2724.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2724.png,COVID/masks/COVID-2724.png,True,False,299,299,L,130.58,e0639a032b5d8908998df3cec5dbdd1a
COVID-2725.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2725.png,COVID/images/COVID-2725.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2725.png,COVID/masks/COVID-2725.png,True,False,299,299,L,137.93,43f221f44ffa14f3320db6bf523c5485
COVID-2726.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2726.png,COVID/images/COVID-2726.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2726.png,COVID/masks/COVID-2726.png,True,False,299,299,L,139.92,9000f7da82b2e3cf5c0488b7ecd90ab5
COVID-2727.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2727.png,COVID/images/COVID-2727.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2727.png,COVID/masks/COVID-2727.png,True,False,299,299,L,140.87,c43c2efa0a29253c0f8b12410029f426
COVID-2728.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2728.png,COVID/images/COVID-2728.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2728.png,COVID/masks/COVID-2728.png,True,False,299,299,L,138.33,ebf5188040a0d10e6d5570df0175610c
COVID-2729.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2729.png,COVID/images/COVID-2729.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2729.png,COVID/masks/COVID-2729.png,True,False,299,299,L,125.75,5a03960ec19df55c1b2a58325a4706f5
COVID-273.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-273.png,COVID/images/COVID-273.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-273.png,COVID/masks/COVID-273.png,True,False,299,299,L,150.75,4997338a8b7ff5d044abd6754396176c
COVID-2730.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2730.png,COVID/images/COVID-2730.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2730.png,COVID/masks/COVID-2730.png,True,False,299,299,L,93.81,7893d2c82bdf57491904f3365a0322a7
COVID-2731.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2731.png,COVID/images/COVID-2731.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2731.png,COVID/masks/COVID-2731.png,True,False,299,299,L,100.68,d0f6fbb9048b09fa3726e9223b97b98c
COVID-2732.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2732.png,COVID/images/COVID-2732.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2732.png,COVID/masks/COVID-2732.png,True,False,299,299,L,168.46,99d3be6d32feaf8e516af82aa35c2917
COVID-2733.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2733.png,COVID/images/COVID-2733.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2733.png,COVID/masks/COVID-2733.png,True,False,299,299,L,144.3,f7c38a1adb64aeb32c5497babf90f808
COVID-2734.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2734.png,COVID/images/COVID-2734.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2734.png,COVID/masks/COVID-2734.png,True,False,299,299,L,143.18,4e1eaf723762dfbfcf893bd7e2e9545f
COVID-2735.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2735.png,COVID/images/COVID-2735.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2735.png,COVID/masks/COVID-2735.png,True,False,299,299,L,172.01,b7c7357ce2bee856cfff91f30103e5fb
COVID-2736.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2736.png,COVID/images/COVID-2736.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2736.png,COVID/masks/COVID-2736.png,True,False,299,299,L,144.18,39dbc66c52b579fc499b6b496219cd72
COVID-2737.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2737.png,COVID/images/COVID-2737.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2737.png,COVID/masks/COVID-2737.png,True,False,299,299,L,145.84,3f995bea01926b7d5f105f2205c68be8
COVID-2738.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2738.png,COVID/images/COVID-2738.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2738.png,COVID/masks/COVID-2738.png,True,False,299,299,L,154.61,e21d9267bc2059132481abc85d28b0e6
COVID-2739.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2739.png,COVID/images/COVID-2739.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2739.png,COVID/masks/COVID-2739.png,True,False,299,299,L,167.69,48e07c93f36bb8860300cc287018bddb
COVID-274.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-274.png,COVID/images/COVID-274.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-274.png,COVID/masks/COVID-274.png,True,False,299,299,L,143.22,885506ad78f0d04271edd65081b68ef8
COVID-2740.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2740.png,COVID/images/COVID-2740.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2740.png,COVID/masks/COVID-2740.png,True,False,299,299,L,172.88,a2766b01dfd2fe11e897fe687ba15ad1
COVID-2741.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2741.png,COVID/images/COVID-2741.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2741.png,COVID/masks/COVID-2741.png,True,False,299,299,L,127.81,95596ced1ec610da36fc9dd5faa153d3
COVID-2742.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2742.png,COVID/images/COVID-2742.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2742.png,COVID/masks/COVID-2742.png,True,False,299,299,L,141.09,369afc4ff7b1df00e9830196a64bbd5a
COVID-2743.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2743.png,COVID/images/COVID-2743.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2743.png,COVID/masks/COVID-2743.png,True,False,299,299,L,76.01,e9189c40c6608f884ba64f383a0110c8
COVID-2744.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2744.png,COVID/images/COVID-2744.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2744.png,COVID/masks/COVID-2744.png,True,False,299,299,L,114.57,b344f68e7fcf1a60abcf30afc62f2398
COVID-2745.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2745.png,COVID/images/COVID-2745.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2745.png,COVID/masks/COVID-2745.png,True,False,299,299,L,171.17,e18e47e655c25963a1302bda6f36078b
COVID-2746.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2746.png,COVID/images/COVID-2746.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2746.png,COVID/masks/COVID-2746.png,True,False,299,299,L,136.04,63e90304944933a2028a58f68d620a88
COVID-2747.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2747.png,COVID/images/COVID-2747.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2747.png,COVID/masks/COVID-2747.png,True,False,299,299,L,159.38,afaa02b18e493af58912ef1d397743b2
COVID-2748.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2748.png,COVID/images/COVID-2748.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2748.png,COVID/masks/COVID-2748.png,True,False,299,299,L,121.44,92dfe512f85bbaee7490b1717a77df7d
COVID-2749.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2749.png,COVID/images/COVID-2749.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2749.png,COVID/masks/COVID-2749.png,True,False,299,299,L,193.06,576adb94dd87dfbe8feb12bd2072b078
COVID-275.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-275.png,COVID/images/COVID-275.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-275.png,COVID/masks/COVID-275.png,True,False,299,299,L,176.46,ba9b6406c05b44ff0d55557f8c3965ef
COVID-2750.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2750.png,COVID/images/COVID-2750.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2750.png,COVID/masks/COVID-2750.png,True,False,299,299,L,110.13,d1fd4965c29151bd6b077895abe40c4f
COVID-2751.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2751.png,COVID/images/COVID-2751.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2751.png,COVID/masks/COVID-2751.png,True,False,299,299,L,102.55,89677af77e4c8c54d838d3a1d02e6327
COVID-2752.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2752.png,COVID/images/COVID-2752.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2752.png,COVID/masks/COVID-2752.png,True,False,299,299,L,134.78,a89b34af1fda9fa133340a80fd1ded91
COVID-2753.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2753.png,COVID/images/COVID-2753.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2753.png,COVID/masks/COVID-2753.png,True,False,299,299,L,179.42,3efd3b2fb3c2e0fded77e5303a8d3109
COVID-2754.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2754.png,COVID/images/COVID-2754.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2754.png,COVID/masks/COVID-2754.png,True,False,299,299,L,139.63,c1f06185a975a43786f69176b0ddf056
COVID-2755.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2755.png,COVID/images/COVID-2755.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2755.png,COVID/masks/COVID-2755.png,True,False,299,299,L,116.97,d83ac6618cde9bbc78194311b04bd1f0
COVID-2756.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2756.png,COVID/images/COVID-2756.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2756.png,COVID/masks/COVID-2756.png,True,False,299,299,L,173.93,ee49e962a792dc5f3e8695d5eb3bfe8e
COVID-2757.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2757.png,COVID/images/COVID-2757.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2757.png,COVID/masks/COVID-2757.png,True,False,299,299,L,143.61,2e0b1f2b79c6b29159757900767676f8
COVID-2758.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2758.png,COVID/images/COVID-2758.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2758.png,COVID/masks/COVID-2758.png,True,False,299,299,L,125.95,00e5a0e242e1e1f29e03c936bd8c49a9
COVID-2759.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2759.png,COVID/images/COVID-2759.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2759.png,COVID/masks/COVID-2759.png,True,False,299,299,L,148.9,519a31e18c0ab1cc0e80b9029cc27b21
COVID-276.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-276.png,COVID/images/COVID-276.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-276.png,COVID/masks/COVID-276.png,True,False,299,299,L,148.69,115aa30450a558aa6af42be5ec88a07a
COVID-2760.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2760.png,COVID/images/COVID-2760.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2760.png,COVID/masks/COVID-2760.png,True,False,299,299,L,137.02,f45ba3d2c4ff471dafffc0a63e4ef9b8
COVID-2761.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2761.png,COVID/images/COVID-2761.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2761.png,COVID/masks/COVID-2761.png,True,False,299,299,L,138.61,ffe01713df8ed79428426f4caffdcfe8
COVID-2762.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2762.png,COVID/images/COVID-2762.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2762.png,COVID/masks/COVID-2762.png,True,False,299,299,L,96.6,68632462f09572d80e4c0762f3e13f7b
COVID-2763.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2763.png,COVID/images/COVID-2763.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2763.png,COVID/masks/COVID-2763.png,True,False,299,299,L,152.17,e1f0daa6dd44f1034e74fcf627301fed
COVID-2764.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2764.png,COVID/images/COVID-2764.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2764.png,COVID/masks/COVID-2764.png,True,False,299,299,L,145.66,444f5101af1a13442ba516bd7d618e15
COVID-2765.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2765.png,COVID/images/COVID-2765.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2765.png,COVID/masks/COVID-2765.png,True,False,299,299,L,151.14,2dd5371a6a10772ff15f60b1bb902d5c
COVID-2766.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2766.png,COVID/images/COVID-2766.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2766.png,COVID/masks/COVID-2766.png,True,False,299,299,L,150.93,554c8136861fcca0546143cd7ff038e7
COVID-2767.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2767.png,COVID/images/COVID-2767.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2767.png,COVID/masks/COVID-2767.png,True,False,299,299,L,188.15,b09feaa4ae074ccf98e88e7c121e18c6
COVID-2768.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2768.png,COVID/images/COVID-2768.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2768.png,COVID/masks/COVID-2768.png,True,False,299,299,L,146.73,d2b0bf8d673cc57b5dc3f290a7f7b503
COVID-2769.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2769.png,COVID/images/COVID-2769.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2769.png,COVID/masks/COVID-2769.png,True,False,299,299,L,124.48,d3830df8d2da8f71614f8d3ec17bf973
COVID-277.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-277.png,COVID/images/COVID-277.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-277.png,COVID/masks/COVID-277.png,True,False,299,299,L,136.78,1a04569e91352c22e74f888d06aace5f
COVID-2770.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2770.png,COVID/images/COVID-2770.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2770.png,COVID/masks/COVID-2770.png,True,False,299,299,L,149.5,fc8164067305eca3360ae2a1e5f6e28d
COVID-2771.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2771.png,COVID/images/COVID-2771.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2771.png,COVID/masks/COVID-2771.png,True,False,299,299,L,127.05,97be8ce6307bd6df80d121bf42b01661
COVID-2772.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2772.png,COVID/images/COVID-2772.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2772.png,COVID/masks/COVID-2772.png,True,False,299,299,L,123.98,fd4fa24f5f5677dd8c886c5171cc171e
COVID-2773.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2773.png,COVID/images/COVID-2773.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2773.png,COVID/masks/COVID-2773.png,True,False,299,299,L,123.98,fd4fa24f5f5677dd8c886c5171cc171e
COVID-2774.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2774.png,COVID/images/COVID-2774.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2774.png,COVID/masks/COVID-2774.png,True,False,299,299,L,123.98,fd4fa24f5f5677dd8c886c5171cc171e
COVID-2775.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2775.png,COVID/images/COVID-2775.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2775.png,COVID/masks/COVID-2775.png,True,False,299,299,L,126.36,6b8789db43e10f1a9843973fc86ee3ce
COVID-2776.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2776.png,COVID/images/COVID-2776.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2776.png,COVID/masks/COVID-2776.png,True,False,299,299,L,138.15,4a60dc2dcc1e9652a4e3398dfcddfcc1
COVID-2777.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2777.png,COVID/images/COVID-2777.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2777.png,COVID/masks/COVID-2777.png,True,False,299,299,L,91.26,3d543f9c73072eb4fca50188972341f8
COVID-2778.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2778.png,COVID/images/COVID-2778.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2778.png,COVID/masks/COVID-2778.png,True,False,299,299,L,137.6,4d5b8487d11941919ea17454aa61323b
COVID-2779.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2779.png,COVID/images/COVID-2779.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2779.png,COVID/masks/COVID-2779.png,True,False,299,299,L,167.65,1dca471fafbd9eba878980ab61002247
COVID-278.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-278.png,COVID/images/COVID-278.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-278.png,COVID/masks/COVID-278.png,True,False,299,299,L,155.1,ca02dab1b8dfce4d5ef48c7e823fd888
COVID-2780.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2780.png,COVID/images/COVID-2780.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2780.png,COVID/masks/COVID-2780.png,True,False,299,299,L,154.04,debecd2bf6d3cf2141c04da5f5368e4e
COVID-2781.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2781.png,COVID/images/COVID-2781.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2781.png,COVID/masks/COVID-2781.png,True,False,299,299,L,164.21,9d22e2fa9e568c39efdcbec415a8063f
COVID-2782.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2782.png,COVID/images/COVID-2782.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2782.png,COVID/masks/COVID-2782.png,True,False,299,299,L,82.51,ad6a54cddc70e95f7dc17672b705758c
COVID-2783.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2783.png,COVID/images/COVID-2783.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2783.png,COVID/masks/COVID-2783.png,True,False,299,299,L,111.14,664c0b3c76e3032adf06175d67864500
COVID-2784.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2784.png,COVID/images/COVID-2784.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2784.png,COVID/masks/COVID-2784.png,True,False,299,299,L,116.51,4da83473f50b298e35e1857d2ca6c8d8
COVID-2785.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2785.png,COVID/images/COVID-2785.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2785.png,COVID/masks/COVID-2785.png,True,False,299,299,L,115.98,11def4a0c3d13190908cccb567a2ad37
COVID-2786.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2786.png,COVID/images/COVID-2786.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2786.png,COVID/masks/COVID-2786.png,True,False,299,299,L,82.7,74bf8b94dd8f2bfeb7dc696d77e828a0
COVID-2787.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2787.png,COVID/images/COVID-2787.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2787.png,COVID/masks/COVID-2787.png,True,False,299,299,L,94.06,4bfebed594deac0cb2f9d8a143226854
COVID-2788.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2788.png,COVID/images/COVID-2788.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2788.png,COVID/masks/COVID-2788.png,True,False,299,299,L,148.72,65aaefafaf15efbac7d154739299bc36
COVID-2789.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2789.png,COVID/images/COVID-2789.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2789.png,COVID/masks/COVID-2789.png,True,False,299,299,L,137.27,cb3949507937c766b9763879182e6ec4
COVID-279.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-279.png,COVID/images/COVID-279.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-279.png,COVID/masks/COVID-279.png,True,False,299,299,L,152.7,a1530f914cc09dada70e7f7c7d1e8a1f
COVID-2790.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2790.png,COVID/images/COVID-2790.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2790.png,COVID/masks/COVID-2790.png,True,False,299,299,L,107.67,f7fc0ee0b411dc30c829b8959ab87435
COVID-2791.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2791.png,COVID/images/COVID-2791.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2791.png,COVID/masks/COVID-2791.png,True,False,299,299,L,144.72,2a21cc98b73eb869e03f8a8829e9ce81
COVID-2792.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2792.png,COVID/images/COVID-2792.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2792.png,COVID/masks/COVID-2792.png,True,False,299,299,L,116.8,2681ac64bec187f3141a3587bcd94dc5
COVID-2793.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2793.png,COVID/images/COVID-2793.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2793.png,COVID/masks/COVID-2793.png,True,False,299,299,L,138.69,deec7920db8958c17925725651663058
COVID-2794.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2794.png,COVID/images/COVID-2794.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2794.png,COVID/masks/COVID-2794.png,True,False,299,299,L,106.45,1121e54324e389d13082a749881d0c9a
COVID-2795.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2795.png,COVID/images/COVID-2795.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2795.png,COVID/masks/COVID-2795.png,True,False,299,299,L,157.98,7602205feb135502201f416cadc214f5
COVID-2796.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2796.png,COVID/images/COVID-2796.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2796.png,COVID/masks/COVID-2796.png,True,False,299,299,L,130.4,1a2a08658698b13a62744a1b728a1922
COVID-2797.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2797.png,COVID/images/COVID-2797.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2797.png,COVID/masks/COVID-2797.png,True,False,299,299,L,175.06,fc51173f70735a0ef045369e72587f80
COVID-2798.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2798.png,COVID/images/COVID-2798.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2798.png,COVID/masks/COVID-2798.png,True,False,299,299,L,132.22,7f29c430800952fd0b37c6d2d929b80b
COVID-2799.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2799.png,COVID/images/COVID-2799.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2799.png,COVID/masks/COVID-2799.png,True,False,299,299,L,126.46,ee2ee0539249fa739798189253c2e68a
COVID-28.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-28.png,COVID/images/COVID-28.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-28.png,COVID/masks/COVID-28.png,True,False,299,299,L,142.62,d0e57fdd44c41de1cf84e95d3567ba3d
COVID-280.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-280.png,COVID/images/COVID-280.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-280.png,COVID/masks/COVID-280.png,True,False,299,299,L,136.78,1a04569e91352c22e74f888d06aace5f
COVID-2800.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2800.png,COVID/images/COVID-2800.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2800.png,COVID/masks/COVID-2800.png,True,False,299,299,L,138.6,d84d8fd2b905d87114907b26324c7049
COVID-2801.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2801.png,COVID/images/COVID-2801.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2801.png,COVID/masks/COVID-2801.png,True,False,299,299,L,121.5,7ac125562f3fa69e46c5d6ed1aefdbab
COVID-2802.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2802.png,COVID/images/COVID-2802.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2802.png,COVID/masks/COVID-2802.png,True,False,299,299,L,115.88,893d52480c1be7896eb44ae1fd0aa2ef
COVID-2803.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2803.png,COVID/images/COVID-2803.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2803.png,COVID/masks/COVID-2803.png,True,False,299,299,L,119.86,0cdf7d597903608f2c4cae5f6314c309
COVID-2804.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2804.png,COVID/images/COVID-2804.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2804.png,COVID/masks/COVID-2804.png,True,False,299,299,L,136.46,86c2accca2b44eaa027e34474ab4fc68
COVID-2805.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2805.png,COVID/images/COVID-2805.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2805.png,COVID/masks/COVID-2805.png,True,False,299,299,L,136.46,86c2accca2b44eaa027e34474ab4fc68
COVID-2806.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2806.png,COVID/images/COVID-2806.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2806.png,COVID/masks/COVID-2806.png,True,False,299,299,L,138.63,f9af3d761249641d4c29c21dcd480927
COVID-2807.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2807.png,COVID/images/COVID-2807.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2807.png,COVID/masks/COVID-2807.png,True,False,299,299,L,137.01,88b2849b8322667577a0d7afea9ec953
COVID-2808.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2808.png,COVID/images/COVID-2808.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2808.png,COVID/masks/COVID-2808.png,True,False,299,299,L,137.01,88b2849b8322667577a0d7afea9ec953
COVID-2809.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2809.png,COVID/images/COVID-2809.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2809.png,COVID/masks/COVID-2809.png,True,False,299,299,L,137.01,c6e5471758bfa2b35fe0d29413e89938
COVID-281.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-281.png,COVID/images/COVID-281.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-281.png,COVID/masks/COVID-281.png,True,False,299,299,L,159.19,fc79bc8ed3f5819446a0822ec741bb3a
COVID-2810.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2810.png,COVID/images/COVID-2810.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2810.png,COVID/masks/COVID-2810.png,True,False,299,299,L,137.01,c6e5471758bfa2b35fe0d29413e89938
COVID-2811.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2811.png,COVID/images/COVID-2811.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2811.png,COVID/masks/COVID-2811.png,True,False,299,299,L,137.01,c6e5471758bfa2b35fe0d29413e89938
COVID-2812.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2812.png,COVID/images/COVID-2812.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2812.png,COVID/masks/COVID-2812.png,True,False,299,299,L,137.01,c6e5471758bfa2b35fe0d29413e89938
COVID-2813.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2813.png,COVID/images/COVID-2813.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2813.png,COVID/masks/COVID-2813.png,True,False,299,299,L,137.01,c6e5471758bfa2b35fe0d29413e89938
COVID-2814.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2814.png,COVID/images/COVID-2814.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2814.png,COVID/masks/COVID-2814.png,True,False,299,299,L,138.5,8cf806f80c77a5dee39ebe16c2109641
COVID-2815.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2815.png,COVID/images/COVID-2815.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2815.png,COVID/masks/COVID-2815.png,True,False,299,299,L,142.72,815c82180379460de9f15146b9670e1b
COVID-2816.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2816.png,COVID/images/COVID-2816.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2816.png,COVID/masks/COVID-2816.png,True,False,299,299,L,130.63,56aeb49ff422f7042b8fd4765dfd42b3
COVID-2817.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2817.png,COVID/images/COVID-2817.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2817.png,COVID/masks/COVID-2817.png,True,False,299,299,L,100.47,b93dc81032d1da5c896ff3d9f2631b1c
COVID-2818.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2818.png,COVID/images/COVID-2818.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2818.png,COVID/masks/COVID-2818.png,True,False,299,299,L,123.2,d63667c5653bfa08f95d65e204689115
COVID-2819.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2819.png,COVID/images/COVID-2819.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2819.png,COVID/masks/COVID-2819.png,True,False,299,299,L,119.87,69875a72f278df2fe41f1a87c89d501e
COVID-282.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-282.png,COVID/images/COVID-282.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-282.png,COVID/masks/COVID-282.png,True,False,299,299,L,171.67,22e9022cfc312ab7071d070276174dce
COVID-2820.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2820.png,COVID/images/COVID-2820.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2820.png,COVID/masks/COVID-2820.png,True,False,299,299,L,154.37,650020ff6fb2c41b50c6dcf6b1a40c97
COVID-2821.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2821.png,COVID/images/COVID-2821.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2821.png,COVID/masks/COVID-2821.png,True,False,299,299,L,152.98,2bad54f5d9f987a6b195cc6d4e2f067e
COVID-2822.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2822.png,COVID/images/COVID-2822.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2822.png,COVID/masks/COVID-2822.png,True,False,299,299,L,146.34,435935c948d0604f08dc00e7f4ec8dbb
COVID-2823.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2823.png,COVID/images/COVID-2823.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2823.png,COVID/masks/COVID-2823.png,True,False,299,299,L,146.34,435935c948d0604f08dc00e7f4ec8dbb
COVID-2824.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2824.png,COVID/images/COVID-2824.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2824.png,COVID/masks/COVID-2824.png,True,False,299,299,L,139.04,0e814a8d26fd0abdd9e711680c48b270
COVID-2825.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2825.png,COVID/images/COVID-2825.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2825.png,COVID/masks/COVID-2825.png,True,False,299,299,L,186.32,40c96a1d129309a01a09704e178bd9a5
COVID-2826.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2826.png,COVID/images/COVID-2826.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2826.png,COVID/masks/COVID-2826.png,True,False,299,299,L,142.5,807008d7facb1431664517448b5667fe
COVID-2827.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2827.png,COVID/images/COVID-2827.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2827.png,COVID/masks/COVID-2827.png,True,False,299,299,L,159.79,016bc4c87c822cc4990af6d24820398a
COVID-2828.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2828.png,COVID/images/COVID-2828.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2828.png,COVID/masks/COVID-2828.png,True,False,299,299,L,110.12,592003d17881edf6bb8ffc3921f93996
COVID-2829.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2829.png,COVID/images/COVID-2829.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2829.png,COVID/masks/COVID-2829.png,True,False,299,299,L,135.52,d36275b3ace27028124e1c6db160a147
COVID-283.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-283.png,COVID/images/COVID-283.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-283.png,COVID/masks/COVID-283.png,True,False,299,299,L,151.01,ef505d244f4b465ae177489c8a595b6f
COVID-2830.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2830.png,COVID/images/COVID-2830.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2830.png,COVID/masks/COVID-2830.png,True,False,299,299,L,138.02,68c6eddfda4d9c9e812c2c89facfe4f5
COVID-2831.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2831.png,COVID/images/COVID-2831.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2831.png,COVID/masks/COVID-2831.png,True,False,299,299,L,143.67,ae60bd52eb29e25a7209e0962e1af760
COVID-2832.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2832.png,COVID/images/COVID-2832.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2832.png,COVID/masks/COVID-2832.png,True,False,299,299,L,148.11,ea2ec0099bd013d58f146e24986377f9
COVID-2833.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2833.png,COVID/images/COVID-2833.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2833.png,COVID/masks/COVID-2833.png,True,False,299,299,L,133.45,1bcfd302a574c1be3dd2dafc99df00be
COVID-2834.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2834.png,COVID/images/COVID-2834.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2834.png,COVID/masks/COVID-2834.png,True,False,299,299,L,103.71,9c2947e3bd7d5abb36f6ac7656fc6473
COVID-2835.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2835.png,COVID/images/COVID-2835.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2835.png,COVID/masks/COVID-2835.png,True,False,299,299,L,155.92,7e8beb58b6d945bce50619b9b571681d
COVID-2836.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2836.png,COVID/images/COVID-2836.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2836.png,COVID/masks/COVID-2836.png,True,False,299,299,L,65.73,b4554a210cce6776edc1f36c211e2986
COVID-2837.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2837.png,COVID/images/COVID-2837.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2837.png,COVID/masks/COVID-2837.png,True,False,299,299,L,114.39,3bc6934c265fb832e4f226b147d83939
COVID-2838.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2838.png,COVID/images/COVID-2838.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2838.png,COVID/masks/COVID-2838.png,True,False,299,299,L,82.84,1064c7ce6a3973c1ee44b73c3c886e47
COVID-2839.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2839.png,COVID/images/COVID-2839.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2839.png,COVID/masks/COVID-2839.png,True,False,299,299,L,101.72,e45724663f15152cdee1679ac3667dcb
COVID-284.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-284.png,COVID/images/COVID-284.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-284.png,COVID/masks/COVID-284.png,True,False,299,299,L,146.53,763013eef54d3c8da6acb9c519a4fc6e
COVID-2840.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2840.png,COVID/images/COVID-2840.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2840.png,COVID/masks/COVID-2840.png,True,False,299,299,L,157.06,eacadb143126b637870b41175ee04ff8
COVID-2841.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2841.png,COVID/images/COVID-2841.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2841.png,COVID/masks/COVID-2841.png,True,False,299,299,L,134.14,5543d98361a6168cb83cdae8819382be
COVID-2842.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2842.png,COVID/images/COVID-2842.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2842.png,COVID/masks/COVID-2842.png,True,False,299,299,L,142.33,4a6da5dda0b449dafaca0fef29154aa2
COVID-2843.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2843.png,COVID/images/COVID-2843.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2843.png,COVID/masks/COVID-2843.png,True,False,299,299,L,136.68,7b8eb71d1427febfa55e6cf02bec4cd1
COVID-2844.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2844.png,COVID/images/COVID-2844.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2844.png,COVID/masks/COVID-2844.png,True,False,299,299,L,113.49,883cf6b9f610d9ac1a1e700272d83d87
COVID-2845.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2845.png,COVID/images/COVID-2845.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2845.png,COVID/masks/COVID-2845.png,True,False,299,299,L,144.4,8fc39bcb85e6466745ee3acd04316420
COVID-2846.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2846.png,COVID/images/COVID-2846.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2846.png,COVID/masks/COVID-2846.png,True,False,299,299,L,123.56,2274d6ae4f97981aae8b1fdd62f1dc5e
COVID-2847.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2847.png,COVID/images/COVID-2847.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2847.png,COVID/masks/COVID-2847.png,True,False,299,299,L,129.35,e139c79a347c80dd77bdddfe0aacada2
COVID-2848.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2848.png,COVID/images/COVID-2848.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2848.png,COVID/masks/COVID-2848.png,True,False,299,299,L,118.38,7e5058a02fac9ee272cb58f9fc73ac15
COVID-2849.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2849.png,COVID/images/COVID-2849.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2849.png,COVID/masks/COVID-2849.png,True,False,299,299,L,107.92,9849b7d7546914dd5845b90279729006
COVID-285.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-285.png,COVID/images/COVID-285.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-285.png,COVID/masks/COVID-285.png,True,False,299,299,L,145.78,455208e9d70022c1174bbbf58b329ecb
COVID-2850.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2850.png,COVID/images/COVID-2850.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2850.png,COVID/masks/COVID-2850.png,True,False,299,299,L,121.15,35f04657e6158a3e78e2f83407eb7093
COVID-2851.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2851.png,COVID/images/COVID-2851.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2851.png,COVID/masks/COVID-2851.png,True,False,299,299,L,127.6,2fc803cbc6d90476f38685b54ec46b72
COVID-2852.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2852.png,COVID/images/COVID-2852.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2852.png,COVID/masks/COVID-2852.png,True,False,299,299,L,158.56,7e2dee9738cf84420344928e97335b45
COVID-2853.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2853.png,COVID/images/COVID-2853.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2853.png,COVID/masks/COVID-2853.png,True,False,299,299,L,102.94,9299153f715d99cadedb4c005b89ae5e
COVID-2854.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2854.png,COVID/images/COVID-2854.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2854.png,COVID/masks/COVID-2854.png,True,False,299,299,L,96.32,dcdc785e02f0a8759a267a3d74f60f07
COVID-2855.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2855.png,COVID/images/COVID-2855.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2855.png,COVID/masks/COVID-2855.png,True,False,299,299,L,150.04,4660075a0585405188b06d7019f5e2ab
COVID-2856.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2856.png,COVID/images/COVID-2856.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2856.png,COVID/masks/COVID-2856.png,True,False,299,299,L,139.74,64de5cf4dfb6ea15ddfba839cedfee85
COVID-2857.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2857.png,COVID/images/COVID-2857.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2857.png,COVID/masks/COVID-2857.png,True,False,299,299,L,128.18,114b7205b3e11e15643dd00b8eac1d83
COVID-2858.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2858.png,COVID/images/COVID-2858.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2858.png,COVID/masks/COVID-2858.png,True,False,299,299,L,126.01,3741a4580fe183dfaf04171c576699a5
COVID-2859.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2859.png,COVID/images/COVID-2859.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2859.png,COVID/masks/COVID-2859.png,True,False,299,299,L,131.15,1b037283f1b132f92702b91648346928
COVID-286.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-286.png,COVID/images/COVID-286.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-286.png,COVID/masks/COVID-286.png,True,False,299,299,L,141.17,6a76420db78261a3c222d8b601bc0908
COVID-2860.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2860.png,COVID/images/COVID-2860.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2860.png,COVID/masks/COVID-2860.png,True,False,299,299,L,81.67,2571453f57085796e97842fa0406fe49
COVID-2861.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2861.png,COVID/images/COVID-2861.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2861.png,COVID/masks/COVID-2861.png,True,False,299,299,L,138.23,75823c02f39b656eabc97d56d2773d98
COVID-2862.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2862.png,COVID/images/COVID-2862.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2862.png,COVID/masks/COVID-2862.png,True,False,299,299,L,131.02,4610a953bf871d3b01bf9236616864a2
COVID-2863.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2863.png,COVID/images/COVID-2863.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2863.png,COVID/masks/COVID-2863.png,True,False,299,299,L,157.84,b4dd5a4f532f45b11ab1d46d138d4870
COVID-2864.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2864.png,COVID/images/COVID-2864.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2864.png,COVID/masks/COVID-2864.png,True,False,299,299,L,174.37,eda19479a756dea6fcf2ec756060c1a0
COVID-2865.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2865.png,COVID/images/COVID-2865.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2865.png,COVID/masks/COVID-2865.png,True,False,299,299,L,167.75,b3403503bc648aa6feb435fd2f52fc95
COVID-2866.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2866.png,COVID/images/COVID-2866.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2866.png,COVID/masks/COVID-2866.png,True,False,299,299,L,177.5,206fb13d517cbe26fc3ebad682a05998
COVID-2867.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2867.png,COVID/images/COVID-2867.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2867.png,COVID/masks/COVID-2867.png,True,False,299,299,L,137.06,67d115390965bb0d55b3bf77b499439a
COVID-2868.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2868.png,COVID/images/COVID-2868.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2868.png,COVID/masks/COVID-2868.png,True,False,299,299,L,138.33,d4f4df2182b84a4245f96296bc963f1b
COVID-2869.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2869.png,COVID/images/COVID-2869.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2869.png,COVID/masks/COVID-2869.png,True,False,299,299,L,155.37,80a0d818a27b1bce8f34323772810642
COVID-287.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-287.png,COVID/images/COVID-287.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-287.png,COVID/masks/COVID-287.png,True,False,299,299,L,147.59,9f6a3086e00f0da2b5b998fbe8513a55
COVID-2870.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2870.png,COVID/images/COVID-2870.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2870.png,COVID/masks/COVID-2870.png,True,False,299,299,L,141.61,95721b9493c8e4db4a03559fd07f3d57
COVID-2871.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2871.png,COVID/images/COVID-2871.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2871.png,COVID/masks/COVID-2871.png,True,False,299,299,L,132.99,8a1bcb2ba7c1e068aab19cd1d82757d2
COVID-2872.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2872.png,COVID/images/COVID-2872.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2872.png,COVID/masks/COVID-2872.png,True,False,299,299,L,130.41,c512ad4df5affe82239c77fef4f2587a
COVID-2873.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2873.png,COVID/images/COVID-2873.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2873.png,COVID/masks/COVID-2873.png,True,False,299,299,L,125.26,9281d87e0a1eb4b472ac11268f21fc8a
COVID-2874.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2874.png,COVID/images/COVID-2874.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2874.png,COVID/masks/COVID-2874.png,True,False,299,299,L,173.36,3e4c30946fc69cf9199884935e644e14
COVID-2875.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2875.png,COVID/images/COVID-2875.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2875.png,COVID/masks/COVID-2875.png,True,False,299,299,L,126.48,00e18849a23ffbc34d4df9be513700d1
COVID-2876.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2876.png,COVID/images/COVID-2876.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2876.png,COVID/masks/COVID-2876.png,True,False,299,299,L,140.5,4a3615c3866b3228c55efbfa994f6457
COVID-2877.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2877.png,COVID/images/COVID-2877.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2877.png,COVID/masks/COVID-2877.png,True,False,299,299,L,122.68,0356170e1be8468968fd8d8cd2f2703a
COVID-2878.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2878.png,COVID/images/COVID-2878.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2878.png,COVID/masks/COVID-2878.png,True,False,299,299,L,156.25,5a59c00d659ede40a41b90af037c1f61
COVID-2879.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2879.png,COVID/images/COVID-2879.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2879.png,COVID/masks/COVID-2879.png,True,False,299,299,L,162.13,c479a603d8956b7d2fe80f43efc6fc02
COVID-288.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-288.png,COVID/images/COVID-288.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-288.png,COVID/masks/COVID-288.png,True,False,299,299,L,154.35,70169f269ce5204d6ab21867539c9ffa
COVID-2880.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2880.png,COVID/images/COVID-2880.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2880.png,COVID/masks/COVID-2880.png,True,False,299,299,L,146.53,d4999f9b164dba68419f72b24f3bf1d0
COVID-2881.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2881.png,COVID/images/COVID-2881.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2881.png,COVID/masks/COVID-2881.png,True,False,299,299,L,148.27,e15fea98d4481efe2edbd6f4dd46fbf1
COVID-2882.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2882.png,COVID/images/COVID-2882.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2882.png,COVID/masks/COVID-2882.png,True,False,299,299,L,133.61,c33524cdda485b9a739f18d4c175db27
COVID-2883.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2883.png,COVID/images/COVID-2883.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2883.png,COVID/masks/COVID-2883.png,True,False,299,299,L,133.65,c0f907433a6884732f1b55b39c4d35d6
COVID-2884.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2884.png,COVID/images/COVID-2884.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2884.png,COVID/masks/COVID-2884.png,True,False,299,299,L,107.97,51632fb6fc60892c953719dbaba5c2d4
COVID-2885.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2885.png,COVID/images/COVID-2885.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2885.png,COVID/masks/COVID-2885.png,True,False,299,299,L,128.79,e45e4977ef276ef3f68a40ff1d2d39f3
COVID-2886.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2886.png,COVID/images/COVID-2886.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2886.png,COVID/masks/COVID-2886.png,True,False,299,299,L,107.42,4ce282a74d3b56f90ba966094840947e
COVID-2887.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2887.png,COVID/images/COVID-2887.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2887.png,COVID/masks/COVID-2887.png,True,False,299,299,L,102.84,9f0ca947b180b13df2e32862f24e6ae5
COVID-2888.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2888.png,COVID/images/COVID-2888.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2888.png,COVID/masks/COVID-2888.png,True,False,299,299,L,156.22,3d1337f9101216d580c45323e8c3a5ab
COVID-2889.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2889.png,COVID/images/COVID-2889.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2889.png,COVID/masks/COVID-2889.png,True,False,299,299,L,138.94,e90503dd2feeff95dce70f069db48cc6
COVID-289.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-289.png,COVID/images/COVID-289.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-289.png,COVID/masks/COVID-289.png,True,False,299,299,L,133.2,2bb97ce2208f894771cdfe0f333a8a68
COVID-2890.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2890.png,COVID/images/COVID-2890.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2890.png,COVID/masks/COVID-2890.png,True,False,299,299,L,144.49,031d29d57f271c314de26a991351c4ce
COVID-2891.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2891.png,COVID/images/COVID-2891.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2891.png,COVID/masks/COVID-2891.png,True,False,299,299,L,147.16,acab3800d038152e6e282421c9ef137e
COVID-2892.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2892.png,COVID/images/COVID-2892.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2892.png,COVID/masks/COVID-2892.png,True,False,299,299,L,124.9,b13bfa8effce8d770c61ed8969d4cb00
COVID-2893.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2893.png,COVID/images/COVID-2893.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2893.png,COVID/masks/COVID-2893.png,True,False,299,299,L,144.79,991b0653fe67b9078fb789b6ec0ddc19
COVID-2894.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2894.png,COVID/images/COVID-2894.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2894.png,COVID/masks/COVID-2894.png,True,False,299,299,L,144.4,58bb5973cbb842f3f0e75f65cba43192
COVID-2895.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2895.png,COVID/images/COVID-2895.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2895.png,COVID/masks/COVID-2895.png,True,False,299,299,L,127.75,0b0f91efebe9de69b91ac51682898042
COVID-2896.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2896.png,COVID/images/COVID-2896.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2896.png,COVID/masks/COVID-2896.png,True,False,299,299,L,146.92,ef1c98ae177beb7c4e0ce03976c54666
COVID-2897.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2897.png,COVID/images/COVID-2897.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2897.png,COVID/masks/COVID-2897.png,True,False,299,299,L,93.22,c300cacbdc4d7ae315d016cf0d467322
COVID-2898.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2898.png,COVID/images/COVID-2898.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2898.png,COVID/masks/COVID-2898.png,True,False,299,299,L,118.22,171a29134b61f63c0a73579905cd8e5a
COVID-2899.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2899.png,COVID/images/COVID-2899.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2899.png,COVID/masks/COVID-2899.png,True,False,299,299,L,107.02,3e96a42848ca990f5ddea523e672b762
COVID-29.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-29.png,COVID/images/COVID-29.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-29.png,COVID/masks/COVID-29.png,True,False,299,299,L,165.82,c6b92de363cd82e4efd585901fba51cb
COVID-290.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-290.png,COVID/images/COVID-290.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-290.png,COVID/masks/COVID-290.png,True,False,299,299,L,155.1,40d94339c2a322d0138b34f524e9b259
COVID-2900.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2900.png,COVID/images/COVID-2900.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2900.png,COVID/masks/COVID-2900.png,True,False,299,299,L,64.26,e6b23052d102ef279b3771ef4e3e69ad
COVID-2901.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2901.png,COVID/images/COVID-2901.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2901.png,COVID/masks/COVID-2901.png,True,False,299,299,L,139.71,b8ced71c263e7de641617596b44e97bf
COVID-2902.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2902.png,COVID/images/COVID-2902.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2902.png,COVID/masks/COVID-2902.png,True,False,299,299,L,151.55,7a71a6d7946fd37068a14c8e0956c876
COVID-2903.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2903.png,COVID/images/COVID-2903.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2903.png,COVID/masks/COVID-2903.png,True,False,299,299,L,161.96,dc554a6bcd69483436a3d74acf6b0610
COVID-2904.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2904.png,COVID/images/COVID-2904.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2904.png,COVID/masks/COVID-2904.png,True,False,299,299,L,150.54,5c45e411bfd95c29ebd3b2dc7c998ec9
COVID-2905.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2905.png,COVID/images/COVID-2905.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2905.png,COVID/masks/COVID-2905.png,True,False,299,299,L,156.56,ea20ab89d1520155c0fde6e033c0ef96
COVID-2906.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2906.png,COVID/images/COVID-2906.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2906.png,COVID/masks/COVID-2906.png,True,False,299,299,L,128.39,f57390cf718828a225d8fe51fdad46d4
COVID-2907.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2907.png,COVID/images/COVID-2907.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2907.png,COVID/masks/COVID-2907.png,True,False,299,299,L,134.71,d4b8feb1e37432089afc28af152d46b0
COVID-2908.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2908.png,COVID/images/COVID-2908.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2908.png,COVID/masks/COVID-2908.png,True,False,299,299,L,152.05,d1b32e3c7c396a5877b93e3a2ab66e7c
COVID-2909.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2909.png,COVID/images/COVID-2909.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2909.png,COVID/masks/COVID-2909.png,True,False,299,299,L,145.51,a3f8929307a55e88138c120d3806afca
COVID-291.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-291.png,COVID/images/COVID-291.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-291.png,COVID/masks/COVID-291.png,True,False,299,299,L,127.75,43843fafeaefc69efa5329728a6654b7
COVID-2910.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2910.png,COVID/images/COVID-2910.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2910.png,COVID/masks/COVID-2910.png,True,False,299,299,L,144.0,642956ce78ad9de0f207e36b3594c816
COVID-2911.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2911.png,COVID/images/COVID-2911.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2911.png,COVID/masks/COVID-2911.png,True,False,299,299,L,153.66,c66b7ffc88465648d2e65d3d4d34ad7b
COVID-2912.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2912.png,COVID/images/COVID-2912.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2912.png,COVID/masks/COVID-2912.png,True,False,299,299,L,146.41,5da278684d3d7c9ede117008ce5b2255
COVID-2913.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2913.png,COVID/images/COVID-2913.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2913.png,COVID/masks/COVID-2913.png,True,False,299,299,L,152.99,7b2b982fa8a3096a2a31f50f05a865ae
COVID-2914.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2914.png,COVID/images/COVID-2914.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2914.png,COVID/masks/COVID-2914.png,True,False,299,299,L,138.16,5d13fe9ed58a48da0d0869781659aa9c
COVID-2915.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2915.png,COVID/images/COVID-2915.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2915.png,COVID/masks/COVID-2915.png,True,False,299,299,L,151.58,1f1d64ae28610ed3f378ab4f9d1ab03b
COVID-2916.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2916.png,COVID/images/COVID-2916.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2916.png,COVID/masks/COVID-2916.png,True,False,299,299,L,131.63,bd834f567c39b0002a88c53fdc54405e
COVID-2917.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2917.png,COVID/images/COVID-2917.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2917.png,COVID/masks/COVID-2917.png,True,False,299,299,L,134.3,6bbcb60c1db108d723c66af0cad2ede6
COVID-2918.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2918.png,COVID/images/COVID-2918.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2918.png,COVID/masks/COVID-2918.png,True,False,299,299,L,139.79,0cf289f00f81c257ae946a13e4035ce8
COVID-2919.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2919.png,COVID/images/COVID-2919.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2919.png,COVID/masks/COVID-2919.png,True,False,299,299,L,169.2,9215a7f5381be7a13020629e01f2a314
COVID-292.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-292.png,COVID/images/COVID-292.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-292.png,COVID/masks/COVID-292.png,True,False,299,299,L,152.99,bf4f4abe008c51642772d8657af56c54
COVID-2920.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2920.png,COVID/images/COVID-2920.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2920.png,COVID/masks/COVID-2920.png,True,False,299,299,L,162.66,a0c009001b0b8e09a4813220772ec346
COVID-2921.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2921.png,COVID/images/COVID-2921.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2921.png,COVID/masks/COVID-2921.png,True,False,299,299,L,151.42,8458f08604826380e5191713e7e8b6e2
COVID-2922.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2922.png,COVID/images/COVID-2922.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2922.png,COVID/masks/COVID-2922.png,True,False,299,299,L,157.54,7acb257ce3594262ceda17d7cec56848
COVID-2923.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2923.png,COVID/images/COVID-2923.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2923.png,COVID/masks/COVID-2923.png,True,False,299,299,L,127.92,e89964260c06a01a201cca08a5d420c7
COVID-2924.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2924.png,COVID/images/COVID-2924.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2924.png,COVID/masks/COVID-2924.png,True,False,299,299,L,144.25,2051a92b3ac0e96721151fbb45825104
COVID-2925.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2925.png,COVID/images/COVID-2925.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2925.png,COVID/masks/COVID-2925.png,True,False,299,299,L,117.1,d6aaa99235d3515bf4d5d38c52a19c6e
COVID-2926.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2926.png,COVID/images/COVID-2926.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2926.png,COVID/masks/COVID-2926.png,True,False,299,299,L,166.52,60c52e2eba452a6588db0bad2b4a351f
COVID-2927.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2927.png,COVID/images/COVID-2927.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2927.png,COVID/masks/COVID-2927.png,True,False,299,299,L,153.67,c54d147b7ea1348c326f45d6f16e6039
COVID-2928.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2928.png,COVID/images/COVID-2928.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2928.png,COVID/masks/COVID-2928.png,True,False,299,299,L,141.03,65677c5865e4f0974f71b8b2bf6c2a1e
COVID-2929.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2929.png,COVID/images/COVID-2929.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2929.png,COVID/masks/COVID-2929.png,True,False,299,299,L,105.13,bc6895bc8b1dd51e54551ae844b2048c
COVID-293.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-293.png,COVID/images/COVID-293.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-293.png,COVID/masks/COVID-293.png,True,False,299,299,L,140.39,2496bac87ad726b59616700e3a76b5df
COVID-2930.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2930.png,COVID/images/COVID-2930.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2930.png,COVID/masks/COVID-2930.png,True,False,299,299,L,104.21,645054ed666151687ce125b83fda1dbf
COVID-2931.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2931.png,COVID/images/COVID-2931.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2931.png,COVID/masks/COVID-2931.png,True,False,299,299,L,125.08,f8ebf1ef83fc1be2ac9c61f798cda6c4
COVID-2932.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2932.png,COVID/images/COVID-2932.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2932.png,COVID/masks/COVID-2932.png,True,False,299,299,L,166.17,a3a318cee9d6338e352d20376983db1b
COVID-2933.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2933.png,COVID/images/COVID-2933.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2933.png,COVID/masks/COVID-2933.png,True,False,299,299,L,138.54,ddafac09b0d658fafdca8260313d044e
COVID-2934.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2934.png,COVID/images/COVID-2934.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2934.png,COVID/masks/COVID-2934.png,True,False,299,299,L,106.07,2ec586d3e1565bbd64db19612d4f7471
COVID-2935.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2935.png,COVID/images/COVID-2935.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2935.png,COVID/masks/COVID-2935.png,True,False,299,299,L,124.93,14d949618b7eb1e85a6a68a0fd4c54bc
COVID-2936.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2936.png,COVID/images/COVID-2936.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2936.png,COVID/masks/COVID-2936.png,True,False,299,299,L,144.53,94ed4f5d820c19e99d6f2c6e6496de33
COVID-2937.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2937.png,COVID/images/COVID-2937.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2937.png,COVID/masks/COVID-2937.png,True,False,299,299,L,149.45,ad605b1e55c998d97c136891d1df9899
COVID-2938.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2938.png,COVID/images/COVID-2938.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2938.png,COVID/masks/COVID-2938.png,True,False,299,299,L,142.17,62e3f06124780b667f6d774d53daa72d
COVID-2939.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2939.png,COVID/images/COVID-2939.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2939.png,COVID/masks/COVID-2939.png,True,False,299,299,L,124.1,e7508db75cba30d2b2c621f0bc23d2d1
COVID-294.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-294.png,COVID/images/COVID-294.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-294.png,COVID/masks/COVID-294.png,True,False,299,299,L,151.01,7192a9906e7df27e6d2f324aade03c3b
COVID-2940.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2940.png,COVID/images/COVID-2940.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2940.png,COVID/masks/COVID-2940.png,True,False,299,299,L,165.96,1c15cf96d66c93c540518aeb3bf9bb98
COVID-2941.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2941.png,COVID/images/COVID-2941.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2941.png,COVID/masks/COVID-2941.png,True,False,299,299,L,126.43,40d4e00a3448fc5d9d095afc990c7253
COVID-2942.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2942.png,COVID/images/COVID-2942.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2942.png,COVID/masks/COVID-2942.png,True,False,299,299,L,126.43,40d4e00a3448fc5d9d095afc990c7253
COVID-2943.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2943.png,COVID/images/COVID-2943.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2943.png,COVID/masks/COVID-2943.png,True,False,299,299,L,134.39,d1511c4c2ca8fa644c4896de857af817
COVID-2944.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2944.png,COVID/images/COVID-2944.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2944.png,COVID/masks/COVID-2944.png,True,False,299,299,L,128.46,ad9d0fc304e80428d9577fc4033a4186
COVID-2945.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2945.png,COVID/images/COVID-2945.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2945.png,COVID/masks/COVID-2945.png,True,False,299,299,L,129.87,d4537e736d1b44bea6eed7750e3ee82e
COVID-2946.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2946.png,COVID/images/COVID-2946.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2946.png,COVID/masks/COVID-2946.png,True,False,299,299,L,129.87,d6a93853a10d9d1b6574abe6cb9295e3
COVID-2947.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2947.png,COVID/images/COVID-2947.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2947.png,COVID/masks/COVID-2947.png,True,False,299,299,L,130.53,a14cf32dffb7d3f3e2bb808e4af39968
COVID-2948.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2948.png,COVID/images/COVID-2948.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2948.png,COVID/masks/COVID-2948.png,True,False,299,299,L,134.77,6fb68cb073ab1c783289b83672750f21
COVID-2949.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2949.png,COVID/images/COVID-2949.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2949.png,COVID/masks/COVID-2949.png,True,False,299,299,L,162.25,09e8e1afec2dd6727ce0082fffb52403
COVID-295.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-295.png,COVID/images/COVID-295.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-295.png,COVID/masks/COVID-295.png,True,False,299,299,L,154.32,44fefb055e28a8aae18981313eac9ef1
COVID-2950.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2950.png,COVID/images/COVID-2950.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2950.png,COVID/masks/COVID-2950.png,True,False,299,299,L,139.88,14f63e19f323d12ca6f06f406a527488
COVID-2951.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2951.png,COVID/images/COVID-2951.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2951.png,COVID/masks/COVID-2951.png,True,False,299,299,L,101.62,f21151db7e220e531300049c609cf220
COVID-2952.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2952.png,COVID/images/COVID-2952.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2952.png,COVID/masks/COVID-2952.png,True,False,299,299,L,103.16,8a01f92b569c1b012b35b20b3b70ce3f
COVID-2953.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2953.png,COVID/images/COVID-2953.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2953.png,COVID/masks/COVID-2953.png,True,False,299,299,L,133.47,9937c5f24a716c8bc0046c674f7bf421
COVID-2954.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2954.png,COVID/images/COVID-2954.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2954.png,COVID/masks/COVID-2954.png,True,False,299,299,L,110.06,afbb0f5216ce365f507c5fd9eeb0703f
COVID-2955.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2955.png,COVID/images/COVID-2955.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2955.png,COVID/masks/COVID-2955.png,True,False,299,299,L,101.07,1c9252a99ac5388495244e63fefd0561
COVID-2956.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2956.png,COVID/images/COVID-2956.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2956.png,COVID/masks/COVID-2956.png,True,False,299,299,L,109.44,27896d5d9a9a8bdd37a8cbe509ae1722
COVID-2957.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2957.png,COVID/images/COVID-2957.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2957.png,COVID/masks/COVID-2957.png,True,False,299,299,L,135.87,7f62b6947dcdc4dbb8df804ac6648758
COVID-2958.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2958.png,COVID/images/COVID-2958.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2958.png,COVID/masks/COVID-2958.png,True,False,299,299,L,137.02,2822c05b4306026c5ec9f8ea3d0735e2
COVID-2959.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2959.png,COVID/images/COVID-2959.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2959.png,COVID/masks/COVID-2959.png,True,False,299,299,L,143.73,61719998727e1fa49bb33fd2400f6078
COVID-296.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-296.png,COVID/images/COVID-296.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-296.png,COVID/masks/COVID-296.png,True,False,299,299,L,171.77,56064163afee2e906f24026fe10120f9
COVID-2960.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2960.png,COVID/images/COVID-2960.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2960.png,COVID/masks/COVID-2960.png,True,False,299,299,L,126.86,9a7b5533d2c9f18d4dd02b720fd0a1d9
COVID-2961.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2961.png,COVID/images/COVID-2961.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2961.png,COVID/masks/COVID-2961.png,True,False,299,299,L,126.49,f77e8fc671e416469458114e431b3733
COVID-2962.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2962.png,COVID/images/COVID-2962.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2962.png,COVID/masks/COVID-2962.png,True,False,299,299,L,129.7,9ef79421bdb289d1ecb6c356e16151a9
COVID-2963.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2963.png,COVID/images/COVID-2963.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2963.png,COVID/masks/COVID-2963.png,True,False,299,299,L,186.32,19aa62fded1a2c22d8fd552598b3cc66
COVID-2964.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2964.png,COVID/images/COVID-2964.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2964.png,COVID/masks/COVID-2964.png,True,False,299,299,L,123.54,d48d9554e48b3275eb2e3a5f1bb20a79
COVID-2965.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2965.png,COVID/images/COVID-2965.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2965.png,COVID/masks/COVID-2965.png,True,False,299,299,L,126.35,5a66054d810c3df20ab1fd82c7cfa10a
COVID-2966.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2966.png,COVID/images/COVID-2966.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2966.png,COVID/masks/COVID-2966.png,True,False,299,299,L,116.98,18acfbd43a8b768684e4a4e21bee6512
COVID-2967.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2967.png,COVID/images/COVID-2967.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2967.png,COVID/masks/COVID-2967.png,True,False,299,299,L,124.87,1cae470168c5337c7b6154297b4a9edd
COVID-2968.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2968.png,COVID/images/COVID-2968.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2968.png,COVID/masks/COVID-2968.png,True,False,299,299,L,124.95,d6958b73b6ec7e0f4ab3f2241fe11184
COVID-2969.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2969.png,COVID/images/COVID-2969.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2969.png,COVID/masks/COVID-2969.png,True,False,299,299,L,114.88,6d3abe0f6f5267a979432a3dac48c60d
COVID-297.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-297.png,COVID/images/COVID-297.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-297.png,COVID/masks/COVID-297.png,True,False,299,299,L,159.31,9a873eab17e9cddd43babc48f61d0db9
COVID-2970.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2970.png,COVID/images/COVID-2970.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2970.png,COVID/masks/COVID-2970.png,True,False,299,299,L,84.63,d4238d58693ff371dab9cd47f29ba09f
COVID-2971.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2971.png,COVID/images/COVID-2971.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2971.png,COVID/masks/COVID-2971.png,True,False,299,299,L,137.13,49ef2ea487fee9938bcd67f2472d6393
COVID-2972.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2972.png,COVID/images/COVID-2972.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2972.png,COVID/masks/COVID-2972.png,True,False,299,299,L,129.14,4d01f081d91579f065b3e21052c8946b
COVID-2973.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2973.png,COVID/images/COVID-2973.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2973.png,COVID/masks/COVID-2973.png,True,False,299,299,L,143.1,411990f3e4d0d3199eebc298a64445dd
COVID-2974.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2974.png,COVID/images/COVID-2974.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2974.png,COVID/masks/COVID-2974.png,True,False,299,299,L,133.88,7586a225541165140d69eea672057d42
COVID-2975.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2975.png,COVID/images/COVID-2975.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2975.png,COVID/masks/COVID-2975.png,True,False,299,299,L,154.49,395fa78f3882b26566715b1765653a47
COVID-2976.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2976.png,COVID/images/COVID-2976.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2976.png,COVID/masks/COVID-2976.png,True,False,299,299,L,160.58,00ea6f3878347cad4b729538421e20d2
COVID-2977.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2977.png,COVID/images/COVID-2977.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2977.png,COVID/masks/COVID-2977.png,True,False,299,299,L,135.95,6c1606bc460c24e5159ce6533618c0f2
COVID-2978.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2978.png,COVID/images/COVID-2978.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2978.png,COVID/masks/COVID-2978.png,True,False,299,299,L,139.44,94dec752dc7d4a93b7a215539158059c
COVID-2979.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2979.png,COVID/images/COVID-2979.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2979.png,COVID/masks/COVID-2979.png,True,False,299,299,L,133.17,a4fdd47b3f7aa9dc7f35da7fe640e440
COVID-298.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-298.png,COVID/images/COVID-298.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-298.png,COVID/masks/COVID-298.png,True,False,299,299,L,152.2,f820d95e21f034ba2ee6810952827b99
COVID-2980.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2980.png,COVID/images/COVID-2980.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2980.png,COVID/masks/COVID-2980.png,True,False,299,299,L,145.94,e3acc9db8e117639d93c5f89e763418b
COVID-2981.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2981.png,COVID/images/COVID-2981.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2981.png,COVID/masks/COVID-2981.png,True,False,299,299,L,148.62,34c7f0aeeda6add62e373413ec10b787
COVID-2982.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2982.png,COVID/images/COVID-2982.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2982.png,COVID/masks/COVID-2982.png,True,False,299,299,L,182.28,19c308733a2bd19e524f492ecd4588eb
COVID-2983.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2983.png,COVID/images/COVID-2983.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2983.png,COVID/masks/COVID-2983.png,True,False,299,299,L,118.15,42eaf8599b30f1b4d7216402c8d7677f
COVID-2984.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2984.png,COVID/images/COVID-2984.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2984.png,COVID/masks/COVID-2984.png,True,False,299,299,L,146.06,8d36bd4d07b73697d7bd68fb4af1ae2e
COVID-2985.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2985.png,COVID/images/COVID-2985.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2985.png,COVID/masks/COVID-2985.png,True,False,299,299,L,141.85,023641d355b981eaac08f70f92350649
COVID-2986.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2986.png,COVID/images/COVID-2986.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2986.png,COVID/masks/COVID-2986.png,True,False,299,299,L,93.35,cd25fadfac71beec3fcc949d68e5d6a1
COVID-2987.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2987.png,COVID/images/COVID-2987.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2987.png,COVID/masks/COVID-2987.png,True,False,299,299,L,110.16,15eaa01327d80ceb74ed9c607ec2c775
COVID-2988.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2988.png,COVID/images/COVID-2988.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2988.png,COVID/masks/COVID-2988.png,True,False,299,299,L,149.16,12f241ce133b5f7beb10eee16b13feab
COVID-2989.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2989.png,COVID/images/COVID-2989.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2989.png,COVID/masks/COVID-2989.png,True,False,299,299,L,162.92,9dd99ee2ed4a4e23c968e6ba3a94530e
COVID-299.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-299.png,COVID/images/COVID-299.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-299.png,COVID/masks/COVID-299.png,True,False,299,299,L,167.91,63e1e2eca69a42e45428e35a4eba95c6
COVID-2990.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2990.png,COVID/images/COVID-2990.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2990.png,COVID/masks/COVID-2990.png,True,False,299,299,L,137.39,af874c48f42424c3031e211926c5dfe9
COVID-2991.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2991.png,COVID/images/COVID-2991.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2991.png,COVID/masks/COVID-2991.png,True,False,299,299,L,153.94,d0b9cef56dfe1c6769f828f95269c451
COVID-2992.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2992.png,COVID/images/COVID-2992.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2992.png,COVID/masks/COVID-2992.png,True,False,299,299,L,142.03,9d3ce12207802a3d4ea2a5ca0a701f3c
COVID-2993.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2993.png,COVID/images/COVID-2993.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2993.png,COVID/masks/COVID-2993.png,True,False,299,299,L,152.3,052a4a06d137e3a979cd3bd243457760
COVID-2994.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2994.png,COVID/images/COVID-2994.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2994.png,COVID/masks/COVID-2994.png,True,False,299,299,L,147.37,a4e559149ceb913e1b6875b5570f8be3
COVID-2995.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2995.png,COVID/images/COVID-2995.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2995.png,COVID/masks/COVID-2995.png,True,False,299,299,L,123.59,52245c09396290ff159990b56d1e0aa0
COVID-2996.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2996.png,COVID/images/COVID-2996.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2996.png,COVID/masks/COVID-2996.png,True,False,299,299,L,149.24,c369a9f2e4588b060d47ebf527290a44
COVID-2997.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2997.png,COVID/images/COVID-2997.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2997.png,COVID/masks/COVID-2997.png,True,False,299,299,L,125.23,093cf56ec91d6b929cd1d462216487a9
COVID-2998.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2998.png,COVID/images/COVID-2998.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2998.png,COVID/masks/COVID-2998.png,True,False,299,299,L,97.55,98ba5f1dcc319675646947f92080aa4e
COVID-2999.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-2999.png,COVID/images/COVID-2999.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-2999.png,COVID/masks/COVID-2999.png,True,False,299,299,L,192.6,6d702d04322c34f49220181c543f569d
COVID-3.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3.png,COVID/images/COVID-3.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3.png,COVID/masks/COVID-3.png,True,False,299,299,L,140.6,a8ec5328e56b07819e3c4f4453a74ba1
COVID-30.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-30.png,COVID/images/COVID-30.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-30.png,COVID/masks/COVID-30.png,True,False,299,299,L,150.91,6ce095cdfd871d2af846ac5eaed7044c
COVID-300.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-300.png,COVID/images/COVID-300.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-300.png,COVID/masks/COVID-300.png,True,False,299,299,L,152.17,c36a3f08ce915ebb73f0e2f0f925598e
COVID-3000.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3000.png,COVID/images/COVID-3000.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3000.png,COVID/masks/COVID-3000.png,True,False,299,299,L,145.71,7340f2a2b873d9b369504a264e25d959
COVID-3001.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3001.png,COVID/images/COVID-3001.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3001.png,COVID/masks/COVID-3001.png,True,False,299,299,L,91.24,1a979a4d20a6441a965fb36faee260d9
COVID-3002.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3002.png,COVID/images/COVID-3002.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3002.png,COVID/masks/COVID-3002.png,True,False,299,299,L,92.54,66aa5cbc8fbf3e6758f66c4078f2e9ae
COVID-3003.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3003.png,COVID/images/COVID-3003.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3003.png,COVID/masks/COVID-3003.png,True,False,299,299,L,143.48,af20febb4c2fa107fc2c0c2bbdfe7a33
COVID-3004.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3004.png,COVID/images/COVID-3004.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3004.png,COVID/masks/COVID-3004.png,True,False,299,299,L,143.03,b8e1fcf1dbda3c5b173f2f2b98d2c53f
COVID-3005.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3005.png,COVID/images/COVID-3005.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3005.png,COVID/masks/COVID-3005.png,True,False,299,299,L,140.98,8984cf098fe1fb838cd1eec7860753ac
COVID-3006.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3006.png,COVID/images/COVID-3006.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3006.png,COVID/masks/COVID-3006.png,True,False,299,299,L,62.13,5e1d36d0761a8d1423acf4f0fa450dfd
COVID-3007.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3007.png,COVID/images/COVID-3007.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3007.png,COVID/masks/COVID-3007.png,True,False,299,299,L,133.16,532acc16a3fb90f85af57922a66f61f5
COVID-3008.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3008.png,COVID/images/COVID-3008.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3008.png,COVID/masks/COVID-3008.png,True,False,299,299,L,140.24,1368a8bca23e124638eb2d4d577069d8
COVID-3009.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3009.png,COVID/images/COVID-3009.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3009.png,COVID/masks/COVID-3009.png,True,False,299,299,L,162.3,0d06274a931d3e29499d1ff084766d31
COVID-301.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-301.png,COVID/images/COVID-301.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-301.png,COVID/masks/COVID-301.png,True,False,299,299,L,150.83,b0390e6be6bc881a33fd712d051bc30b
COVID-3010.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3010.png,COVID/images/COVID-3010.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3010.png,COVID/masks/COVID-3010.png,True,False,299,299,L,139.31,d426b0517f23a5437a905fab467a3d9e
COVID-3011.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3011.png,COVID/images/COVID-3011.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3011.png,COVID/masks/COVID-3011.png,True,False,299,299,L,134.87,5f2798d7be362357f809536c22aaa465
COVID-3012.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3012.png,COVID/images/COVID-3012.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3012.png,COVID/masks/COVID-3012.png,True,False,299,299,L,113.4,637b8a8ed84117b90428b59657da7682
COVID-3013.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3013.png,COVID/images/COVID-3013.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3013.png,COVID/masks/COVID-3013.png,True,False,299,299,L,131.59,600da708fd34d698504d09382312b1ff
COVID-3014.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3014.png,COVID/images/COVID-3014.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3014.png,COVID/masks/COVID-3014.png,True,False,299,299,L,158.47,4f7a8602611b38fb58273b3e993255fb
COVID-3015.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3015.png,COVID/images/COVID-3015.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3015.png,COVID/masks/COVID-3015.png,True,False,299,299,L,117.3,bce95436a02f16a226a6742ff74c2bb8
COVID-3016.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3016.png,COVID/images/COVID-3016.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3016.png,COVID/masks/COVID-3016.png,True,False,299,299,L,140.08,11f7ed84095819fdbf73857bf05e6bd8
COVID-3017.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3017.png,COVID/images/COVID-3017.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3017.png,COVID/masks/COVID-3017.png,True,False,299,299,L,143.98,7d4ca00c37651d54b967a43a035409b7
COVID-3018.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3018.png,COVID/images/COVID-3018.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3018.png,COVID/masks/COVID-3018.png,True,False,299,299,L,167.34,5338174c2576e0d4b0e1c2dc5105f8c8
COVID-3019.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3019.png,COVID/images/COVID-3019.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3019.png,COVID/masks/COVID-3019.png,True,False,299,299,L,118.4,0415cd6ad63d19b84a50ae14f5f6c945
COVID-302.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-302.png,COVID/images/COVID-302.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-302.png,COVID/masks/COVID-302.png,True,False,299,299,L,134.74,4f73540da04f1b18a0d5e8ab7849779c
COVID-3020.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3020.png,COVID/images/COVID-3020.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3020.png,COVID/masks/COVID-3020.png,True,False,299,299,L,119.05,e94110d705ab242c7514ca6121e85bae
COVID-3021.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3021.png,COVID/images/COVID-3021.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3021.png,COVID/masks/COVID-3021.png,True,False,299,299,L,145.81,9131accf6ee7226073afd462638f65df
COVID-3022.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3022.png,COVID/images/COVID-3022.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3022.png,COVID/masks/COVID-3022.png,True,False,299,299,L,131.8,d46eaff756567b2bfff5b18401ff85fc
COVID-3023.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3023.png,COVID/images/COVID-3023.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3023.png,COVID/masks/COVID-3023.png,True,False,299,299,L,158.74,21a8704617c45415f4f97f8b0e2f3d31
COVID-3024.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3024.png,COVID/images/COVID-3024.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3024.png,COVID/masks/COVID-3024.png,True,False,299,299,L,138.29,1307918df043666c249a258bf3f8925d
COVID-3025.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3025.png,COVID/images/COVID-3025.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3025.png,COVID/masks/COVID-3025.png,True,False,299,299,L,158.93,1824545735d528c5bf4bbb3f8bd0ba73
COVID-3026.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3026.png,COVID/images/COVID-3026.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3026.png,COVID/masks/COVID-3026.png,True,False,299,299,L,120.93,26b9bdb43798517099c242b1e5c1b349
COVID-3027.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3027.png,COVID/images/COVID-3027.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3027.png,COVID/masks/COVID-3027.png,True,False,299,299,L,166.11,5aa2e29387260c9da18d9bc6c0227810
COVID-3028.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3028.png,COVID/images/COVID-3028.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3028.png,COVID/masks/COVID-3028.png,True,False,299,299,L,141.91,6f238efaf78b44d3bb49a34fa09678f3
COVID-3029.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3029.png,COVID/images/COVID-3029.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3029.png,COVID/masks/COVID-3029.png,True,False,299,299,L,105.27,a5507f7426f03d44e06ef38c0bccc998
COVID-303.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-303.png,COVID/images/COVID-303.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-303.png,COVID/masks/COVID-303.png,True,False,299,299,L,160.45,bcda21b3feeae521fd66cdb60f9f9ddc
COVID-3030.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3030.png,COVID/images/COVID-3030.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3030.png,COVID/masks/COVID-3030.png,True,False,299,299,L,117.4,2b423499189a410af6937909947d9d00
COVID-3031.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3031.png,COVID/images/COVID-3031.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3031.png,COVID/masks/COVID-3031.png,True,False,299,299,L,104.5,1c6de62e7bff90b46fdc4c9912fdfcf0
COVID-3032.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3032.png,COVID/images/COVID-3032.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3032.png,COVID/masks/COVID-3032.png,True,False,299,299,L,156.96,ac5d5c49b4d74e5bca62d21270bf42fd
COVID-3033.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3033.png,COVID/images/COVID-3033.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3033.png,COVID/masks/COVID-3033.png,True,False,299,299,L,137.59,a3708f5d697d96533dfca08116173624
COVID-3034.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3034.png,COVID/images/COVID-3034.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3034.png,COVID/masks/COVID-3034.png,True,False,299,299,L,131.85,160b65941d2abd5de5dc653d13fa9fde
COVID-3035.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3035.png,COVID/images/COVID-3035.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3035.png,COVID/masks/COVID-3035.png,True,False,299,299,L,155.29,163448d55dab6e257718c960841570f8
COVID-3036.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3036.png,COVID/images/COVID-3036.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3036.png,COVID/masks/COVID-3036.png,True,False,299,299,L,129.49,9f7d3a64347967155ec4c277da7b248a
COVID-3037.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3037.png,COVID/images/COVID-3037.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3037.png,COVID/masks/COVID-3037.png,True,False,299,299,L,128.31,33db32a7922bffdd34de19ddc4ada20a
COVID-3038.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3038.png,COVID/images/COVID-3038.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3038.png,COVID/masks/COVID-3038.png,True,False,299,299,L,137.11,0b915a51b2445e50f43dc3587cbd4ce7
COVID-3039.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3039.png,COVID/images/COVID-3039.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3039.png,COVID/masks/COVID-3039.png,True,False,299,299,L,75.27,4418c3d4c863fcc785031188bbf1af19
COVID-304.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-304.png,COVID/images/COVID-304.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-304.png,COVID/masks/COVID-304.png,True,False,299,299,L,159.49,0d23ea6b9ba08e20d117d864d02ba4a5
COVID-3040.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3040.png,COVID/images/COVID-3040.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3040.png,COVID/masks/COVID-3040.png,True,False,299,299,L,123.15,5ba5bffee55d9b89391c277bd0765f9e
COVID-3041.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3041.png,COVID/images/COVID-3041.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3041.png,COVID/masks/COVID-3041.png,True,False,299,299,L,141.63,caaef051fa8421ac7201744c6b599851
COVID-3042.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3042.png,COVID/images/COVID-3042.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3042.png,COVID/masks/COVID-3042.png,True,False,299,299,L,136.59,a6739a38ff77ac1af8c077729c475c2f
COVID-3043.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3043.png,COVID/images/COVID-3043.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3043.png,COVID/masks/COVID-3043.png,True,False,299,299,L,174.98,ccd3aac3e787310c4c212941188c6662
COVID-3044.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3044.png,COVID/images/COVID-3044.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3044.png,COVID/masks/COVID-3044.png,True,False,299,299,L,174.88,00ed8725167b04f87bf58207bcd917aa
COVID-3045.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3045.png,COVID/images/COVID-3045.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3045.png,COVID/masks/COVID-3045.png,True,False,299,299,L,103.96,2283813d1072a465f940eeba2004faaa
COVID-3046.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3046.png,COVID/images/COVID-3046.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3046.png,COVID/masks/COVID-3046.png,True,False,299,299,L,151.63,ab1b17b098e68a00d7bfbe24b39b2db2
COVID-3047.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3047.png,COVID/images/COVID-3047.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3047.png,COVID/masks/COVID-3047.png,True,False,299,299,L,133.66,55d614eb1411541bda16da825210bbc8
COVID-3048.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3048.png,COVID/images/COVID-3048.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3048.png,COVID/masks/COVID-3048.png,True,False,299,299,L,113.28,a3a6dfaf34bc326d9fa76c5926d8b9e2
COVID-3049.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3049.png,COVID/images/COVID-3049.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3049.png,COVID/masks/COVID-3049.png,True,False,299,299,L,162.06,03091cf4c471bdc2ec80802f71260be6
COVID-305.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-305.png,COVID/images/COVID-305.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-305.png,COVID/masks/COVID-305.png,True,False,299,299,L,155.76,4d6f1618a2148595bdadc3bde3f49f33
COVID-3050.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3050.png,COVID/images/COVID-3050.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3050.png,COVID/masks/COVID-3050.png,True,False,299,299,L,128.33,71e43738168d64172a2d51926ee45f36
COVID-3051.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3051.png,COVID/images/COVID-3051.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3051.png,COVID/masks/COVID-3051.png,True,False,299,299,L,134.3,ae02eecbbaa6915c04c080cd4394f34d
COVID-3052.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3052.png,COVID/images/COVID-3052.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3052.png,COVID/masks/COVID-3052.png,True,False,299,299,L,134.64,c1fb64896f0f164c44509866b6a91af0
COVID-3053.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3053.png,COVID/images/COVID-3053.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3053.png,COVID/masks/COVID-3053.png,True,False,299,299,L,120.42,c17e918c8e1fe1656e6ad4553554e086
COVID-3054.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3054.png,COVID/images/COVID-3054.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3054.png,COVID/masks/COVID-3054.png,True,False,299,299,L,135.16,cb79847cb86f3b22921e0b3abbbe839d
COVID-3055.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3055.png,COVID/images/COVID-3055.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3055.png,COVID/masks/COVID-3055.png,True,False,299,299,L,169.88,99896af214a90cdb755a48f5c4f9d02f
COVID-3056.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3056.png,COVID/images/COVID-3056.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3056.png,COVID/masks/COVID-3056.png,True,False,299,299,L,150.59,c4e74d473a35b6ee52253533819eeb6d
COVID-3057.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3057.png,COVID/images/COVID-3057.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3057.png,COVID/masks/COVID-3057.png,True,False,299,299,L,95.06,b08f549f9eda3232fd88d553dfd2a8b6
COVID-3058.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3058.png,COVID/images/COVID-3058.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3058.png,COVID/masks/COVID-3058.png,True,False,299,299,L,124.53,9fd28ce4344d33f6b0101b5c83d930a5
COVID-3059.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3059.png,COVID/images/COVID-3059.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3059.png,COVID/masks/COVID-3059.png,True,False,299,299,L,151.82,c06d2f8c16f13fe176df6b10d4dbc92c
COVID-306.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-306.png,COVID/images/COVID-306.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-306.png,COVID/masks/COVID-306.png,True,False,299,299,L,155.97,76be90a3b3542df139642d47b9535a60
COVID-3060.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3060.png,COVID/images/COVID-3060.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3060.png,COVID/masks/COVID-3060.png,True,False,299,299,L,147.48,2db0a5d8e3d60ae21ae285381ea8bd32
COVID-3061.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3061.png,COVID/images/COVID-3061.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3061.png,COVID/masks/COVID-3061.png,True,False,299,299,L,165.28,1e0c6784605f46abbe5d80bdcba19831
COVID-3062.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3062.png,COVID/images/COVID-3062.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3062.png,COVID/masks/COVID-3062.png,True,False,299,299,L,162.24,1ec159ac6e41cf17a5dafd38ff95eaf9
COVID-3063.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3063.png,COVID/images/COVID-3063.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3063.png,COVID/masks/COVID-3063.png,True,False,299,299,L,169.68,25d51f8f241636af77e070cb4e8c5fe8
COVID-3064.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3064.png,COVID/images/COVID-3064.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3064.png,COVID/masks/COVID-3064.png,True,False,299,299,L,163.5,b7328886d1089e44bcc1c8378214010d
COVID-3065.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3065.png,COVID/images/COVID-3065.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3065.png,COVID/masks/COVID-3065.png,True,False,299,299,L,111.37,ce8266676951916a08b2517e5e5041c9
COVID-3066.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3066.png,COVID/images/COVID-3066.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3066.png,COVID/masks/COVID-3066.png,True,False,299,299,L,90.38,f14085bd5b4c0109b2de2d5e51a2ecdf
COVID-3067.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3067.png,COVID/images/COVID-3067.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3067.png,COVID/masks/COVID-3067.png,True,False,299,299,L,149.93,b32cc01a4ca4f57e7c6ca736bba7d121
COVID-3068.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3068.png,COVID/images/COVID-3068.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3068.png,COVID/masks/COVID-3068.png,True,False,299,299,L,147.22,51889981bcfac61f145aab0cf1ffea05
COVID-3069.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3069.png,COVID/images/COVID-3069.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3069.png,COVID/masks/COVID-3069.png,True,False,299,299,L,174.04,38f320b3ab4cf1afbe901bad7e30e9d0
COVID-307.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-307.png,COVID/images/COVID-307.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-307.png,COVID/masks/COVID-307.png,True,False,299,299,L,178.93,2fd1d767129825734f2a665f9eba129a
COVID-3070.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3070.png,COVID/images/COVID-3070.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3070.png,COVID/masks/COVID-3070.png,True,False,299,299,L,126.65,5014199f153dfe41be224b53e1dac5c2
COVID-3071.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3071.png,COVID/images/COVID-3071.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3071.png,COVID/masks/COVID-3071.png,True,False,299,299,L,82.43,a1f8d6c7d174830cf6b6581b1b6c4371
COVID-3072.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3072.png,COVID/images/COVID-3072.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3072.png,COVID/masks/COVID-3072.png,True,False,299,299,L,129.82,8cb3b55e0d127dbe7c8e37e10d4323ee
COVID-3073.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3073.png,COVID/images/COVID-3073.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3073.png,COVID/masks/COVID-3073.png,True,False,299,299,L,157.92,6ca465355c2d36ea5bf4c46b058488eb
COVID-3074.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3074.png,COVID/images/COVID-3074.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3074.png,COVID/masks/COVID-3074.png,True,False,299,299,L,130.09,977795f1213bbfbdecb23e004a5f025f
COVID-3075.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3075.png,COVID/images/COVID-3075.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3075.png,COVID/masks/COVID-3075.png,True,False,299,299,L,104.37,f5d274ccdf24f877068d4555f154b582
COVID-3076.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3076.png,COVID/images/COVID-3076.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3076.png,COVID/masks/COVID-3076.png,True,False,299,299,L,134.16,da3e0c4946b6377f6ae3e9114538f0fa
COVID-3077.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3077.png,COVID/images/COVID-3077.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3077.png,COVID/masks/COVID-3077.png,True,False,299,299,L,131.5,80320a65a6b8134c8ee30789acfefd77
COVID-3078.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3078.png,COVID/images/COVID-3078.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3078.png,COVID/masks/COVID-3078.png,True,False,299,299,L,136.78,c0821b679b417db13aab2d01592fa637
COVID-3079.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3079.png,COVID/images/COVID-3079.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3079.png,COVID/masks/COVID-3079.png,True,False,299,299,L,115.87,cf519415000d904e3a4b7a1115925f3b
COVID-308.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-308.png,COVID/images/COVID-308.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-308.png,COVID/masks/COVID-308.png,True,False,299,299,L,129.79,b4fa0689e874ab33df1f8cc15ae34588
COVID-3080.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3080.png,COVID/images/COVID-3080.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3080.png,COVID/masks/COVID-3080.png,True,False,299,299,L,100.23,485bb5f08df63649cbc6a5bde5b954c1
COVID-3081.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3081.png,COVID/images/COVID-3081.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3081.png,COVID/masks/COVID-3081.png,True,False,299,299,L,133.26,8c1ad69a496a0593012064ca81585760
COVID-3082.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3082.png,COVID/images/COVID-3082.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3082.png,COVID/masks/COVID-3082.png,True,False,299,299,L,122.92,a2f677ff33e104522170b35d7d39873a
COVID-3083.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3083.png,COVID/images/COVID-3083.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3083.png,COVID/masks/COVID-3083.png,True,False,299,299,L,118.86,39d48629750ac11ceed5da581b0967cb
COVID-3084.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3084.png,COVID/images/COVID-3084.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3084.png,COVID/masks/COVID-3084.png,True,False,299,299,L,153.27,6d0620ba98411774500bd0cf631a94dd
COVID-3085.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3085.png,COVID/images/COVID-3085.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3085.png,COVID/masks/COVID-3085.png,True,False,299,299,L,157.81,5ee9aa21cc80018280efc024b220a79d
COVID-3086.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3086.png,COVID/images/COVID-3086.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3086.png,COVID/masks/COVID-3086.png,True,False,299,299,L,135.4,584915dc30a62016300f73eff1af21f1
COVID-3087.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3087.png,COVID/images/COVID-3087.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3087.png,COVID/masks/COVID-3087.png,True,False,299,299,L,135.58,8043d66073ccc2b33e56733681247b95
COVID-3088.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3088.png,COVID/images/COVID-3088.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3088.png,COVID/masks/COVID-3088.png,True,False,299,299,L,103.28,e3dfb43703c681addedc8ebde080acd6
COVID-3089.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3089.png,COVID/images/COVID-3089.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3089.png,COVID/masks/COVID-3089.png,True,False,299,299,L,56.69,ac1cae6b1bbb0855544ccb7ead7db9aa
COVID-309.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-309.png,COVID/images/COVID-309.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-309.png,COVID/masks/COVID-309.png,True,False,299,299,L,142.55,e30698a7f2a9467f65fcbf79562c0e89
COVID-3090.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3090.png,COVID/images/COVID-3090.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3090.png,COVID/masks/COVID-3090.png,True,False,299,299,L,103.87,636ad16394648d4d30b6330b6200d51b
COVID-3091.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3091.png,COVID/images/COVID-3091.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3091.png,COVID/masks/COVID-3091.png,True,False,299,299,L,152.19,8b3861d1a8ebc8cb95da344d12654fa0
COVID-3092.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3092.png,COVID/images/COVID-3092.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3092.png,COVID/masks/COVID-3092.png,True,False,299,299,L,100.6,c525d1cf89818d962e66e9772fee0541
COVID-3093.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3093.png,COVID/images/COVID-3093.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3093.png,COVID/masks/COVID-3093.png,True,False,299,299,L,153.58,7946ad182e11111d2a6bce1d3e981373
COVID-3094.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3094.png,COVID/images/COVID-3094.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3094.png,COVID/masks/COVID-3094.png,True,False,299,299,L,137.76,e7fd5ff4ec48d2f746b6afab35518ace
COVID-3095.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3095.png,COVID/images/COVID-3095.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3095.png,COVID/masks/COVID-3095.png,True,False,299,299,L,162.7,f4747ef08ccbd81b0b05b3340991e2ee
COVID-3096.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3096.png,COVID/images/COVID-3096.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3096.png,COVID/masks/COVID-3096.png,True,False,299,299,L,76.72,9757c4eeefa1d5da9d299bd6bcb5b3d7
COVID-3097.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3097.png,COVID/images/COVID-3097.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3097.png,COVID/masks/COVID-3097.png,True,False,299,299,L,132.67,b586fa87ff9ebad91ad2bd6fcb8e169d
COVID-3098.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3098.png,COVID/images/COVID-3098.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3098.png,COVID/masks/COVID-3098.png,True,False,299,299,L,128.9,a93e3e486112317306e99e40165e358b
COVID-3099.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3099.png,COVID/images/COVID-3099.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3099.png,COVID/masks/COVID-3099.png,True,False,299,299,L,163.35,6732caaac655d7ffad3040b8e8e440f5
COVID-31.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-31.png,COVID/images/COVID-31.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-31.png,COVID/masks/COVID-31.png,True,False,299,299,L,145.17,6b9aaa56834e2bdfa880e27fbf1d8f33
COVID-310.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-310.png,COVID/images/COVID-310.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-310.png,COVID/masks/COVID-310.png,True,False,299,299,L,161.62,ddede401ca016b502a06778b02d8f624
COVID-3100.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3100.png,COVID/images/COVID-3100.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3100.png,COVID/masks/COVID-3100.png,True,False,299,299,L,137.27,7d4c780562f4b952afe3662cb458dee5
COVID-3101.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3101.png,COVID/images/COVID-3101.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3101.png,COVID/masks/COVID-3101.png,True,False,299,299,L,147.83,9f6b614dc7909343f7733f09321debf6
COVID-3102.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3102.png,COVID/images/COVID-3102.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3102.png,COVID/masks/COVID-3102.png,True,False,299,299,L,101.02,1d11e2381c97275cb706dd1782acae82
COVID-3103.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3103.png,COVID/images/COVID-3103.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3103.png,COVID/masks/COVID-3103.png,True,False,299,299,L,96.24,54f74acd17497439d64f050fe70b71d4
COVID-3104.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3104.png,COVID/images/COVID-3104.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3104.png,COVID/masks/COVID-3104.png,True,False,299,299,L,114.21,7f5e1283a934e091d6e5ff9f9256b18b
COVID-3105.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3105.png,COVID/images/COVID-3105.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3105.png,COVID/masks/COVID-3105.png,True,False,299,299,L,73.97,6d49b3537391771aeb660483d728ddc8
COVID-3106.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3106.png,COVID/images/COVID-3106.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3106.png,COVID/masks/COVID-3106.png,True,False,299,299,L,174.74,ab12bc7fdd75e052ce0230766e3f77d9
COVID-3107.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3107.png,COVID/images/COVID-3107.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3107.png,COVID/masks/COVID-3107.png,True,False,299,299,L,95.9,2f780d313ee97fa7b994433e24530963
COVID-3108.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3108.png,COVID/images/COVID-3108.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3108.png,COVID/masks/COVID-3108.png,True,False,299,299,L,126.99,d8d45cfaf1f9f77cc7891f8a2d7860fe
COVID-3109.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3109.png,COVID/images/COVID-3109.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3109.png,COVID/masks/COVID-3109.png,True,False,299,299,L,146.54,9119a5e499466b5ade01ae74163dabff
COVID-311.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-311.png,COVID/images/COVID-311.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-311.png,COVID/masks/COVID-311.png,True,False,299,299,L,169.19,9fdd77128cd0abba3b692f20a7408f5f
COVID-3110.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3110.png,COVID/images/COVID-3110.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3110.png,COVID/masks/COVID-3110.png,True,False,299,299,L,105.15,a5b90c7bde277323943a458966719cec
COVID-3111.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3111.png,COVID/images/COVID-3111.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3111.png,COVID/masks/COVID-3111.png,True,False,299,299,L,134.53,a7d8ebf364a377479d03db77b9e8417b
COVID-3112.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3112.png,COVID/images/COVID-3112.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3112.png,COVID/masks/COVID-3112.png,True,False,299,299,L,100.82,7f8726e5d13bccc6a4e8eb0226e7fa88
COVID-3113.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3113.png,COVID/images/COVID-3113.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3113.png,COVID/masks/COVID-3113.png,True,False,299,299,L,141.52,16795bb1b3f093cde58f7b6fb3bcdfeb
COVID-3114.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3114.png,COVID/images/COVID-3114.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3114.png,COVID/masks/COVID-3114.png,True,False,299,299,L,126.89,3035c2bf6d66b3a1553d38f487aafea2
COVID-3115.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3115.png,COVID/images/COVID-3115.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3115.png,COVID/masks/COVID-3115.png,True,False,299,299,L,149.38,9a789d51c3f9e619c68c2670f5c0ac1d
COVID-3116.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3116.png,COVID/images/COVID-3116.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3116.png,COVID/masks/COVID-3116.png,True,False,299,299,L,83.12,9d76aff96e1ada48c980ec4331780b9c
COVID-3117.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3117.png,COVID/images/COVID-3117.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3117.png,COVID/masks/COVID-3117.png,True,False,299,299,L,85.54,948f5719c0f9de942e93eff086894e59
COVID-3118.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3118.png,COVID/images/COVID-3118.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3118.png,COVID/masks/COVID-3118.png,True,False,299,299,L,142.05,cf98de46db897e484324aafbd3d4f480
COVID-3119.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3119.png,COVID/images/COVID-3119.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3119.png,COVID/masks/COVID-3119.png,True,False,299,299,L,127.35,05610e88c29e24a0e9816799fd098a56
COVID-312.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-312.png,COVID/images/COVID-312.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-312.png,COVID/masks/COVID-312.png,True,False,299,299,L,158.73,86ff0c25cde98021e37e1df47d0b1fa6
COVID-3120.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3120.png,COVID/images/COVID-3120.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3120.png,COVID/masks/COVID-3120.png,True,False,299,299,L,156.03,2f22616c2bbe160f8c4bdbee7c809415
COVID-3121.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3121.png,COVID/images/COVID-3121.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3121.png,COVID/masks/COVID-3121.png,True,False,299,299,L,144.17,03412b4a8e1a7bb5d18603a6e1dd829f
COVID-3122.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3122.png,COVID/images/COVID-3122.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3122.png,COVID/masks/COVID-3122.png,True,False,299,299,L,157.93,9dd758edde82c57c5220598f13fb706f
COVID-3123.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3123.png,COVID/images/COVID-3123.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3123.png,COVID/masks/COVID-3123.png,True,False,299,299,L,149.98,9775e0d7a3e333e1c8fef88300b24bac
COVID-3124.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3124.png,COVID/images/COVID-3124.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3124.png,COVID/masks/COVID-3124.png,True,False,299,299,L,153.79,01877a5cfa225ae4a1273ad9bccd8384
COVID-3125.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3125.png,COVID/images/COVID-3125.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3125.png,COVID/masks/COVID-3125.png,True,False,299,299,L,137.49,e93f297188934152214a3c90967045d5
COVID-3126.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3126.png,COVID/images/COVID-3126.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3126.png,COVID/masks/COVID-3126.png,True,False,299,299,L,130.46,2c23909042ac469d688da29f4f175c2d
COVID-3127.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3127.png,COVID/images/COVID-3127.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3127.png,COVID/masks/COVID-3127.png,True,False,299,299,L,149.44,77b37332980e9ad4cc52bb8d00afe29e
COVID-3128.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3128.png,COVID/images/COVID-3128.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3128.png,COVID/masks/COVID-3128.png,True,False,299,299,L,122.11,29ab43ff793099612efd823d21fc7ace
COVID-3129.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3129.png,COVID/images/COVID-3129.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3129.png,COVID/masks/COVID-3129.png,True,False,299,299,L,123.79,af1e48a30eadd3a1ec71a37fbd32932d
COVID-313.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-313.png,COVID/images/COVID-313.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-313.png,COVID/masks/COVID-313.png,True,False,299,299,L,158.91,366914112af725432c599216e24fa1b8
COVID-3130.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3130.png,COVID/images/COVID-3130.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3130.png,COVID/masks/COVID-3130.png,True,False,299,299,L,135.6,a92c1f6b9c1c47a95d36d950200c13f3
COVID-3131.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3131.png,COVID/images/COVID-3131.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3131.png,COVID/masks/COVID-3131.png,True,False,299,299,L,168.61,320f6dd77940b84ace7328fc46cf0cf3
COVID-3132.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3132.png,COVID/images/COVID-3132.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3132.png,COVID/masks/COVID-3132.png,True,False,299,299,L,124.08,85f9f9ac5ab271f1e9c51d68acd96e0b
COVID-3133.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3133.png,COVID/images/COVID-3133.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3133.png,COVID/masks/COVID-3133.png,True,False,299,299,L,131.34,4e26f263316cb4643e71c3a5ef552d0f
COVID-3134.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3134.png,COVID/images/COVID-3134.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3134.png,COVID/masks/COVID-3134.png,True,False,299,299,L,93.41,7940b3f0878ea1d7ad08c07a9f05bc19
COVID-3135.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3135.png,COVID/images/COVID-3135.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3135.png,COVID/masks/COVID-3135.png,True,False,299,299,L,96.44,68b27d7da4452651a5b9c6b4407b283f
COVID-3136.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3136.png,COVID/images/COVID-3136.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3136.png,COVID/masks/COVID-3136.png,True,False,299,299,L,98.16,dcafb812c0bb0b8de7e4379ea99cdd3a
COVID-3137.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3137.png,COVID/images/COVID-3137.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3137.png,COVID/masks/COVID-3137.png,True,False,299,299,L,98.64,a8838bbb1ace15e45c9d17db9f2b823b
COVID-3138.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3138.png,COVID/images/COVID-3138.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3138.png,COVID/masks/COVID-3138.png,True,False,299,299,L,83.98,3121b55374e88e344ce9349fb6b89167
COVID-3139.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3139.png,COVID/images/COVID-3139.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3139.png,COVID/masks/COVID-3139.png,True,False,299,299,L,141.46,0e6e37a31498350efec0289a28bdd219
COVID-314.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-314.png,COVID/images/COVID-314.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-314.png,COVID/masks/COVID-314.png,True,False,299,299,L,165.25,047006c840f320a138f48f4b5671206f
COVID-3140.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3140.png,COVID/images/COVID-3140.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3140.png,COVID/masks/COVID-3140.png,True,False,299,299,L,142.26,8e7368933d400defbe24f3481ab80b44
COVID-3141.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3141.png,COVID/images/COVID-3141.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3141.png,COVID/masks/COVID-3141.png,True,False,299,299,L,142.26,8e7368933d400defbe24f3481ab80b44
COVID-3142.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3142.png,COVID/images/COVID-3142.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3142.png,COVID/masks/COVID-3142.png,True,False,299,299,L,142.26,8e7368933d400defbe24f3481ab80b44
COVID-3143.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3143.png,COVID/images/COVID-3143.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3143.png,COVID/masks/COVID-3143.png,True,False,299,299,L,142.26,8e7368933d400defbe24f3481ab80b44
COVID-3144.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3144.png,COVID/images/COVID-3144.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3144.png,COVID/masks/COVID-3144.png,True,False,299,299,L,142.26,8e7368933d400defbe24f3481ab80b44
COVID-3145.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3145.png,COVID/images/COVID-3145.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3145.png,COVID/masks/COVID-3145.png,True,False,299,299,L,147.24,5f6e74e46059743b084261aa68594f4c
COVID-3146.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3146.png,COVID/images/COVID-3146.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3146.png,COVID/masks/COVID-3146.png,True,False,299,299,L,130.18,11b381a778ec4320c32cf92209ad6521
COVID-3147.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3147.png,COVID/images/COVID-3147.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3147.png,COVID/masks/COVID-3147.png,True,False,299,299,L,129.63,d71a26955732d01624b1e28a628804c8
COVID-3148.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3148.png,COVID/images/COVID-3148.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3148.png,COVID/masks/COVID-3148.png,True,False,299,299,L,139.09,b478204cb31e954c8d4a9603e1ffa38a
COVID-3149.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3149.png,COVID/images/COVID-3149.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3149.png,COVID/masks/COVID-3149.png,True,False,299,299,L,101.41,94b515faa65cf55ef9aadab0ff4c3595
COVID-315.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-315.png,COVID/images/COVID-315.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-315.png,COVID/masks/COVID-315.png,True,False,299,299,L,151.48,f5e6e6b493f4ec4282b4e0d02e4c8626
COVID-3150.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3150.png,COVID/images/COVID-3150.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3150.png,COVID/masks/COVID-3150.png,True,False,299,299,L,110.21,6b9177d4ac4de239bffb5554eb69ce4c
COVID-3151.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3151.png,COVID/images/COVID-3151.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3151.png,COVID/masks/COVID-3151.png,True,False,299,299,L,98.13,ebb089b6477afceb12b435ccd4db67df
COVID-3152.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3152.png,COVID/images/COVID-3152.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3152.png,COVID/masks/COVID-3152.png,True,False,299,299,L,132.28,8157ac98e431ae73139ea5e1128c4730
COVID-3153.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3153.png,COVID/images/COVID-3153.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3153.png,COVID/masks/COVID-3153.png,True,False,299,299,L,135.46,ed00d0685867e18de52456fe8280f76c
COVID-3154.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3154.png,COVID/images/COVID-3154.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3154.png,COVID/masks/COVID-3154.png,True,False,299,299,L,134.92,01f2b4021880e18ec8842a47655a9d75
COVID-3155.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3155.png,COVID/images/COVID-3155.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3155.png,COVID/masks/COVID-3155.png,True,False,299,299,L,137.39,595e1f8487d3c985b4e6785144474639
COVID-3156.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3156.png,COVID/images/COVID-3156.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3156.png,COVID/masks/COVID-3156.png,True,False,299,299,L,85.8,f4240bce4de9539f3c148054515e9ffc
COVID-3157.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3157.png,COVID/images/COVID-3157.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3157.png,COVID/masks/COVID-3157.png,True,False,299,299,L,157.18,9469eae6e4fbb9c86b5f533321528632
COVID-3158.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3158.png,COVID/images/COVID-3158.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3158.png,COVID/masks/COVID-3158.png,True,False,299,299,L,63.58,75f164246d74405506c6ff93b44f149f
COVID-3159.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3159.png,COVID/images/COVID-3159.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3159.png,COVID/masks/COVID-3159.png,True,False,299,299,L,140.66,80c777fee548dff3254ca7dba7b0729c
COVID-316.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-316.png,COVID/images/COVID-316.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-316.png,COVID/masks/COVID-316.png,True,False,299,299,L,158.38,177eb9750442d0f97874fbc48aabff0c
COVID-3160.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3160.png,COVID/images/COVID-3160.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3160.png,COVID/masks/COVID-3160.png,True,False,299,299,L,119.31,e8941d5eb45e57a5eb225895bd0a74cc
COVID-3161.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3161.png,COVID/images/COVID-3161.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3161.png,COVID/masks/COVID-3161.png,True,False,299,299,L,143.21,b88e320d52b5490345750dba180d88f3
COVID-3162.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3162.png,COVID/images/COVID-3162.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3162.png,COVID/masks/COVID-3162.png,True,False,299,299,L,83.11,5646d528cdfa1b3aa3b7ec62a03ad84c
COVID-3163.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3163.png,COVID/images/COVID-3163.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3163.png,COVID/masks/COVID-3163.png,True,False,299,299,L,79.33,e6e964526a544224b211b736dcc0cf9e
COVID-3164.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3164.png,COVID/images/COVID-3164.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3164.png,COVID/masks/COVID-3164.png,True,False,299,299,L,93.0,421f000de09e84121e61997faee59084
COVID-3165.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3165.png,COVID/images/COVID-3165.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3165.png,COVID/masks/COVID-3165.png,True,False,299,299,L,90.4,d4fbd71047c7dd0b717c1701eab67a9c
COVID-3166.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3166.png,COVID/images/COVID-3166.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3166.png,COVID/masks/COVID-3166.png,True,False,299,299,L,141.07,da0d3eb45fcd35f95a7b5e574616e25f
COVID-3167.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3167.png,COVID/images/COVID-3167.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3167.png,COVID/masks/COVID-3167.png,True,False,299,299,L,148.82,bfa349dd314abfcdeafda66563fd2400
COVID-3168.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3168.png,COVID/images/COVID-3168.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3168.png,COVID/masks/COVID-3168.png,True,False,299,299,L,114.67,8c9503acd4557c43cbcf217a81b22f44
COVID-3169.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3169.png,COVID/images/COVID-3169.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3169.png,COVID/masks/COVID-3169.png,True,False,299,299,L,186.52,418b57cc6dfed8b0132cabca54576568
COVID-317.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-317.png,COVID/images/COVID-317.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-317.png,COVID/masks/COVID-317.png,True,False,299,299,L,161.6,ee037baa29ad7d8bfe9bd8c3deca0bee
COVID-3170.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3170.png,COVID/images/COVID-3170.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3170.png,COVID/masks/COVID-3170.png,True,False,299,299,L,170.25,971ea424bbccb2204e1996541bcd7622
COVID-3171.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3171.png,COVID/images/COVID-3171.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3171.png,COVID/masks/COVID-3171.png,True,False,299,299,L,59.89,975f3b033ab28aeb1f8b18b64e6a2b5f
COVID-3172.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3172.png,COVID/images/COVID-3172.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3172.png,COVID/masks/COVID-3172.png,True,False,299,299,L,140.27,ee0cd1261aeacf83d1f1f66a47e10bf6
COVID-3173.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3173.png,COVID/images/COVID-3173.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3173.png,COVID/masks/COVID-3173.png,True,False,299,299,L,149.73,59d3ade2ade794a72344dbc4f693b477
COVID-3174.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3174.png,COVID/images/COVID-3174.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3174.png,COVID/masks/COVID-3174.png,True,False,299,299,L,157.24,0e292be26c7b1331c9adfe81f6366fc7
COVID-3175.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3175.png,COVID/images/COVID-3175.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3175.png,COVID/masks/COVID-3175.png,True,False,299,299,L,132.0,cffca4f9dccc7226f19f06ee54ca4017
COVID-3176.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3176.png,COVID/images/COVID-3176.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3176.png,COVID/masks/COVID-3176.png,True,False,299,299,L,145.05,ea11ca2f9dee590efb23b3878edb8270
COVID-3177.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3177.png,COVID/images/COVID-3177.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3177.png,COVID/masks/COVID-3177.png,True,False,299,299,L,137.46,eaef6b186bb8370070ac9f6498631cf8
COVID-3178.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3178.png,COVID/images/COVID-3178.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3178.png,COVID/masks/COVID-3178.png,True,False,299,299,L,120.17,cfbc299aaba318e06a97f755001d42b0
COVID-3179.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3179.png,COVID/images/COVID-3179.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3179.png,COVID/masks/COVID-3179.png,True,False,299,299,L,103.33,26551e94140b1462abb819338fadb3d2
COVID-318.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-318.png,COVID/images/COVID-318.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-318.png,COVID/masks/COVID-318.png,True,False,299,299,L,150.61,b519765276e3ff16099c3b416a321a7a
COVID-3180.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3180.png,COVID/images/COVID-3180.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3180.png,COVID/masks/COVID-3180.png,True,False,299,299,L,94.64,58810275efacafbb8dcc6f8efd50c8a9
COVID-3181.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3181.png,COVID/images/COVID-3181.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3181.png,COVID/masks/COVID-3181.png,True,False,299,299,L,153.84,e6a2bc6ff664de1881b14fc6a48c452a
COVID-3182.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3182.png,COVID/images/COVID-3182.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3182.png,COVID/masks/COVID-3182.png,True,False,299,299,L,126.0,0c7b60e199170e4fe2944ee4e4255210
COVID-3183.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3183.png,COVID/images/COVID-3183.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3183.png,COVID/masks/COVID-3183.png,True,False,299,299,L,107.01,c44e00c7d77aa202327f847e20f0f231
COVID-3184.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3184.png,COVID/images/COVID-3184.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3184.png,COVID/masks/COVID-3184.png,True,False,299,299,L,132.0,fb607ffe3e3e1f9be36217e5da89f9ee
COVID-3185.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3185.png,COVID/images/COVID-3185.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3185.png,COVID/masks/COVID-3185.png,True,False,299,299,L,112.95,9965ef1216604952a7548ceae1775b3d
COVID-3186.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3186.png,COVID/images/COVID-3186.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3186.png,COVID/masks/COVID-3186.png,True,False,299,299,L,90.93,9dcda580ae6b4034bed95dec1e7e362e
COVID-3187.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3187.png,COVID/images/COVID-3187.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3187.png,COVID/masks/COVID-3187.png,True,False,299,299,L,77.36,20c6b65d4d02725e7be434b93790f1c4
COVID-3188.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3188.png,COVID/images/COVID-3188.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3188.png,COVID/masks/COVID-3188.png,True,False,299,299,L,149.47,58c4a0c06483c78adaa6ba137337dbf2
COVID-3189.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3189.png,COVID/images/COVID-3189.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3189.png,COVID/masks/COVID-3189.png,True,False,299,299,L,132.51,89c68999d07820a2f8ac28deb8bdd005
COVID-319.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-319.png,COVID/images/COVID-319.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-319.png,COVID/masks/COVID-319.png,True,False,299,299,L,173.02,d9674072f8d9180b149d77dca9ab21b0
COVID-3190.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3190.png,COVID/images/COVID-3190.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3190.png,COVID/masks/COVID-3190.png,True,False,299,299,L,138.46,de379b4e870c632e08e0c3822f5bf11b
COVID-3191.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3191.png,COVID/images/COVID-3191.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3191.png,COVID/masks/COVID-3191.png,True,False,299,299,L,146.44,8701c72e9e6159c8068d22f7e6d23168
COVID-3192.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3192.png,COVID/images/COVID-3192.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3192.png,COVID/masks/COVID-3192.png,True,False,299,299,L,161.41,4981b87223cf98eb49acb73d424219cd
COVID-3193.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3193.png,COVID/images/COVID-3193.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3193.png,COVID/masks/COVID-3193.png,True,False,299,299,L,156.9,336b2260b0eadadc16af4c23267a4301
COVID-3194.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3194.png,COVID/images/COVID-3194.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3194.png,COVID/masks/COVID-3194.png,True,False,299,299,L,169.23,b271fabdf90010ef4c476b2732695cfa
COVID-3195.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3195.png,COVID/images/COVID-3195.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3195.png,COVID/masks/COVID-3195.png,True,False,299,299,L,159.63,40d0f6eee7232be0ba583ab19ae49971
COVID-3196.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3196.png,COVID/images/COVID-3196.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3196.png,COVID/masks/COVID-3196.png,True,False,299,299,L,136.31,cf14dce9b75a85f442d126ca8c543a3d
COVID-3197.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3197.png,COVID/images/COVID-3197.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3197.png,COVID/masks/COVID-3197.png,True,False,299,299,L,140.32,2962c9d011443559bcda5f8fd7a11f21
COVID-3198.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3198.png,COVID/images/COVID-3198.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3198.png,COVID/masks/COVID-3198.png,True,False,299,299,L,115.32,1ee496658c07ca51bef7381d1ae4f5d2
COVID-3199.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3199.png,COVID/images/COVID-3199.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3199.png,COVID/masks/COVID-3199.png,True,False,299,299,L,162.89,1b5c859e64f804a97d8852a55e946a94
COVID-32.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-32.png,COVID/images/COVID-32.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-32.png,COVID/masks/COVID-32.png,True,False,299,299,L,124.47,38762702a3b185bab98ce09df30b5d95
COVID-320.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-320.png,COVID/images/COVID-320.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-320.png,COVID/masks/COVID-320.png,True,False,299,299,L,162.36,2cde7aeb53610802060bcd79d9f61f0e
COVID-3200.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3200.png,COVID/images/COVID-3200.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3200.png,COVID/masks/COVID-3200.png,True,False,299,299,L,134.8,dd1eeb60291e2fded13fefb3c7e7f9ca
COVID-3201.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3201.png,COVID/images/COVID-3201.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3201.png,COVID/masks/COVID-3201.png,True,False,299,299,L,52.5,67231402d202b80c06471743eae1f3bb
COVID-3202.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3202.png,COVID/images/COVID-3202.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3202.png,COVID/masks/COVID-3202.png,True,False,299,299,L,116.18,528bd178da09daa969795d9a57d57acb
COVID-3203.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3203.png,COVID/images/COVID-3203.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3203.png,COVID/masks/COVID-3203.png,True,False,299,299,L,170.75,a7263a8c749888565a4033f35802b4ef
COVID-3204.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3204.png,COVID/images/COVID-3204.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3204.png,COVID/masks/COVID-3204.png,True,False,299,299,L,152.93,4b78961d2c6a3b761a3081f4c30bb664
COVID-3205.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3205.png,COVID/images/COVID-3205.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3205.png,COVID/masks/COVID-3205.png,True,False,299,299,L,166.58,b5323ce82caba5ee82cf0af91426d351
COVID-3206.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3206.png,COVID/images/COVID-3206.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3206.png,COVID/masks/COVID-3206.png,True,False,299,299,L,161.73,b685ecbec81355043f2e9bea55844adb
COVID-3207.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3207.png,COVID/images/COVID-3207.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3207.png,COVID/masks/COVID-3207.png,True,False,299,299,L,163.66,91fc14a1600056eeac4a16fe41f4e010
COVID-3208.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3208.png,COVID/images/COVID-3208.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3208.png,COVID/masks/COVID-3208.png,True,False,299,299,L,163.66,91fc14a1600056eeac4a16fe41f4e010
COVID-3209.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3209.png,COVID/images/COVID-3209.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3209.png,COVID/masks/COVID-3209.png,True,False,299,299,L,124.92,3962f9961ed6e11d63a40c32446380d7
COVID-321.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-321.png,COVID/images/COVID-321.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-321.png,COVID/masks/COVID-321.png,True,False,299,299,L,173.02,d9674072f8d9180b149d77dca9ab21b0
COVID-3210.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3210.png,COVID/images/COVID-3210.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3210.png,COVID/masks/COVID-3210.png,True,False,299,299,L,110.32,3912c8f32c3e39cd7596cdafda56a443
COVID-3211.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3211.png,COVID/images/COVID-3211.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3211.png,COVID/masks/COVID-3211.png,True,False,299,299,L,135.07,c6e8c0b08993a076255cb0bc975f997f
COVID-3212.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3212.png,COVID/images/COVID-3212.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3212.png,COVID/masks/COVID-3212.png,True,False,299,299,L,135.07,c6e8c0b08993a076255cb0bc975f997f
COVID-3213.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3213.png,COVID/images/COVID-3213.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3213.png,COVID/masks/COVID-3213.png,True,False,299,299,L,117.19,1ab669f5dd28ea0f92ec90609f20eb86
COVID-3214.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3214.png,COVID/images/COVID-3214.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3214.png,COVID/masks/COVID-3214.png,True,False,299,299,L,125.4,514eb8eac2e691199962d0fc93447851
COVID-3215.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3215.png,COVID/images/COVID-3215.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3215.png,COVID/masks/COVID-3215.png,True,False,299,299,L,162.96,f151c50b4ed17b85575bffc6e5821423
COVID-3216.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3216.png,COVID/images/COVID-3216.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3216.png,COVID/masks/COVID-3216.png,True,False,299,299,L,155.69,d2fa1e255580574fa6d3a9d29c017bc8
COVID-3217.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3217.png,COVID/images/COVID-3217.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3217.png,COVID/masks/COVID-3217.png,True,False,299,299,L,153.87,3a21b646020438a8ce4fce8cbfa4ef9c
COVID-3218.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3218.png,COVID/images/COVID-3218.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3218.png,COVID/masks/COVID-3218.png,True,False,299,299,L,102.72,eab244cb3d3c8fa6d0cc1966a4222387
COVID-3219.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3219.png,COVID/images/COVID-3219.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3219.png,COVID/masks/COVID-3219.png,True,False,299,299,L,105.65,3956f53056d68b4a16e7a7522e03d4c5
COVID-322.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-322.png,COVID/images/COVID-322.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-322.png,COVID/masks/COVID-322.png,True,False,299,299,L,178.0,ae89f1c412aecae6927e411711147e43
COVID-3220.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3220.png,COVID/images/COVID-3220.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3220.png,COVID/masks/COVID-3220.png,True,False,299,299,L,151.97,1ce60aa4a7624be5f928b8201a9f4574
COVID-3221.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3221.png,COVID/images/COVID-3221.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3221.png,COVID/masks/COVID-3221.png,True,False,299,299,L,137.98,9104257b3b7527caf39817bb1109c0ee
COVID-3222.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3222.png,COVID/images/COVID-3222.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3222.png,COVID/masks/COVID-3222.png,True,False,299,299,L,130.95,97f0927dc25c65b007df3650e50b919d
COVID-3223.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3223.png,COVID/images/COVID-3223.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3223.png,COVID/masks/COVID-3223.png,True,False,299,299,L,130.95,97f0927dc25c65b007df3650e50b919d
COVID-3224.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3224.png,COVID/images/COVID-3224.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3224.png,COVID/masks/COVID-3224.png,True,False,299,299,L,134.8,d6561819bc4c72ccb4e3a1e7160c4bac
COVID-3225.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3225.png,COVID/images/COVID-3225.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3225.png,COVID/masks/COVID-3225.png,True,False,299,299,L,151.73,d3d533c6dc1fcc8db8057d144e142340
COVID-3226.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3226.png,COVID/images/COVID-3226.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3226.png,COVID/masks/COVID-3226.png,True,False,299,299,L,156.71,d11a575692e87bbb42fb9b65d19867e1
COVID-3227.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3227.png,COVID/images/COVID-3227.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3227.png,COVID/masks/COVID-3227.png,True,False,299,299,L,164.66,a5ff7df0bcbce8d66ffc31000135d6df
COVID-3228.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3228.png,COVID/images/COVID-3228.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3228.png,COVID/masks/COVID-3228.png,True,False,299,299,L,94.14,2f4da6bd82f65be7020dafd665c3fb76
COVID-3229.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3229.png,COVID/images/COVID-3229.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3229.png,COVID/masks/COVID-3229.png,True,False,299,299,L,141.89,fa1bcb1ae745ff4ac4d738e432b5d029
COVID-323.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-323.png,COVID/images/COVID-323.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-323.png,COVID/masks/COVID-323.png,True,False,299,299,L,164.71,a2aaa91edd10d32cffb77f624581f55b
COVID-3230.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3230.png,COVID/images/COVID-3230.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3230.png,COVID/masks/COVID-3230.png,True,False,299,299,L,100.29,3edc7119264ce1d37f52b217bd79a2e2
COVID-3231.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3231.png,COVID/images/COVID-3231.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3231.png,COVID/masks/COVID-3231.png,True,False,299,299,L,129.61,4829243cb6e6527b8db870b1ca532bfb
COVID-3232.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3232.png,COVID/images/COVID-3232.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3232.png,COVID/masks/COVID-3232.png,True,False,299,299,L,114.48,904a50e77ff157cb8122ef1f8e07af4c
COVID-3233.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3233.png,COVID/images/COVID-3233.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3233.png,COVID/masks/COVID-3233.png,True,False,299,299,L,86.3,178ab0ce5231fe3a62ff40e9e33885cc
COVID-3234.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3234.png,COVID/images/COVID-3234.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3234.png,COVID/masks/COVID-3234.png,True,False,299,299,L,111.95,7c4829ce1323c4b4ee36a8952d0478b1
COVID-3235.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3235.png,COVID/images/COVID-3235.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3235.png,COVID/masks/COVID-3235.png,True,False,299,299,L,81.95,555b6a59688ae5185e7205f289fb4454
COVID-3236.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3236.png,COVID/images/COVID-3236.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3236.png,COVID/masks/COVID-3236.png,True,False,299,299,L,111.18,665505b0ccc380386bdfae934514e26e
COVID-3237.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3237.png,COVID/images/COVID-3237.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3237.png,COVID/masks/COVID-3237.png,True,False,299,299,L,106.73,48dd860a26c0b7e1aae87d3ba348d150
COVID-3238.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3238.png,COVID/images/COVID-3238.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3238.png,COVID/masks/COVID-3238.png,True,False,299,299,L,112.35,0424d9ef30c71782ad493819ef7ab207
COVID-3239.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3239.png,COVID/images/COVID-3239.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3239.png,COVID/masks/COVID-3239.png,True,False,299,299,L,137.61,36f85801f8c71b5a38721af4dd13fbd8
COVID-324.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-324.png,COVID/images/COVID-324.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-324.png,COVID/masks/COVID-324.png,True,False,299,299,L,171.77,3c214fd59610a4d30d52ca842b457781
COVID-3240.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3240.png,COVID/images/COVID-3240.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3240.png,COVID/masks/COVID-3240.png,True,False,299,299,L,141.72,2707fb3c23433720d0037319a978a754
COVID-3241.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3241.png,COVID/images/COVID-3241.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3241.png,COVID/masks/COVID-3241.png,True,False,299,299,L,159.88,d9636fc73d3c8ee19197a111fa8a98bb
COVID-3242.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3242.png,COVID/images/COVID-3242.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3242.png,COVID/masks/COVID-3242.png,True,False,299,299,L,145.88,767127ec1534ac1aa1b59c5cf9b290a4
COVID-3243.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3243.png,COVID/images/COVID-3243.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3243.png,COVID/masks/COVID-3243.png,True,False,299,299,L,163.27,0a65bdc7661be8560306e62beb8deb66
COVID-3244.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3244.png,COVID/images/COVID-3244.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3244.png,COVID/masks/COVID-3244.png,True,False,299,299,L,139.17,41fc37b05c741f42e0d28a758fcbaf9c
COVID-3245.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3245.png,COVID/images/COVID-3245.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3245.png,COVID/masks/COVID-3245.png,True,False,299,299,L,127.32,a8a9b9c2c144e6c899fb7b68586502c3
COVID-3246.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3246.png,COVID/images/COVID-3246.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3246.png,COVID/masks/COVID-3246.png,True,False,299,299,L,146.75,0e3c7125b4b3d7e4060817c217401390
COVID-3247.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3247.png,COVID/images/COVID-3247.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3247.png,COVID/masks/COVID-3247.png,True,False,299,299,L,122.8,793a69b1244d10425842fde6c374a6d5
COVID-3248.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3248.png,COVID/images/COVID-3248.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3248.png,COVID/masks/COVID-3248.png,True,False,299,299,L,107.14,09bb882064a5db95cde4d7cd555d8259
COVID-3249.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3249.png,COVID/images/COVID-3249.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3249.png,COVID/masks/COVID-3249.png,True,False,299,299,L,111.73,01a278bd56a197838df1bb685823deec
COVID-325.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-325.png,COVID/images/COVID-325.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-325.png,COVID/masks/COVID-325.png,True,False,299,299,L,142.24,c596461ab08a66aee526b1c1f6c6bdca
COVID-3250.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3250.png,COVID/images/COVID-3250.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3250.png,COVID/masks/COVID-3250.png,True,False,299,299,L,169.43,de7d1e1faa1e5ecf0e9f234d23e1a2db
COVID-3251.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3251.png,COVID/images/COVID-3251.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3251.png,COVID/masks/COVID-3251.png,True,False,299,299,L,150.23,67e29b97607eb41fa673e8d301ff01e0
COVID-3252.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3252.png,COVID/images/COVID-3252.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3252.png,COVID/masks/COVID-3252.png,True,False,299,299,L,146.03,81ad893fd2149d638af76db092764738
COVID-3253.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3253.png,COVID/images/COVID-3253.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3253.png,COVID/masks/COVID-3253.png,True,False,299,299,L,121.19,aeed05f8582feeb134e964985bab687d
COVID-3254.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3254.png,COVID/images/COVID-3254.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3254.png,COVID/masks/COVID-3254.png,True,False,299,299,L,157.31,abf2306196980696233a1abf4f29a3ea
COVID-3255.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3255.png,COVID/images/COVID-3255.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3255.png,COVID/masks/COVID-3255.png,True,False,299,299,L,131.0,de351c975b49d10d117846ac396bb048
COVID-3256.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3256.png,COVID/images/COVID-3256.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3256.png,COVID/masks/COVID-3256.png,True,False,299,299,L,126.11,61c1b46ee2813b139d6bec3fec69d2d2
COVID-3257.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3257.png,COVID/images/COVID-3257.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3257.png,COVID/masks/COVID-3257.png,True,False,299,299,L,127.75,8bd3b1f8cf8fbcdd7b9e7b56df0148c8
COVID-3258.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3258.png,COVID/images/COVID-3258.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3258.png,COVID/masks/COVID-3258.png,True,False,299,299,L,138.54,0f1d1f1650c38f360b3cbbf156c0486a
COVID-3259.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3259.png,COVID/images/COVID-3259.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3259.png,COVID/masks/COVID-3259.png,True,False,299,299,L,66.61,4b38c3f8cb8bd0f27b4c0d79af254903
COVID-326.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-326.png,COVID/images/COVID-326.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-326.png,COVID/masks/COVID-326.png,True,False,299,299,L,144.77,b7970a747e74cfdb2ad43d517fc6916d
COVID-3260.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3260.png,COVID/images/COVID-3260.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3260.png,COVID/masks/COVID-3260.png,True,False,299,299,L,106.5,5cd686aa304052f2fd8c88be58035aa9
COVID-3261.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3261.png,COVID/images/COVID-3261.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3261.png,COVID/masks/COVID-3261.png,True,False,299,299,L,143.67,52cdbb3ff6c39e6fbb704b45e4b9dd38
COVID-3262.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3262.png,COVID/images/COVID-3262.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3262.png,COVID/masks/COVID-3262.png,True,False,299,299,L,157.38,0430a36f899329e144effe459b422c92
COVID-3263.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3263.png,COVID/images/COVID-3263.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3263.png,COVID/masks/COVID-3263.png,True,False,299,299,L,93.46,a58a2534cc4acf58d8a7113840a28fb5
COVID-3264.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3264.png,COVID/images/COVID-3264.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3264.png,COVID/masks/COVID-3264.png,True,False,299,299,L,106.0,d7a53831d136775102722930f727f64f
COVID-3265.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3265.png,COVID/images/COVID-3265.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3265.png,COVID/masks/COVID-3265.png,True,False,299,299,L,122.92,600b04c48fa724a73f6a38fd35ad99c6
COVID-3266.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3266.png,COVID/images/COVID-3266.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3266.png,COVID/masks/COVID-3266.png,True,False,299,299,L,66.91,8cad43eb7506953b65711afc23a25498
COVID-3267.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3267.png,COVID/images/COVID-3267.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3267.png,COVID/masks/COVID-3267.png,True,False,299,299,L,63.94,7b11571b1d8ac8a6199f42fa3b4c5e5b
COVID-3268.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3268.png,COVID/images/COVID-3268.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3268.png,COVID/masks/COVID-3268.png,True,False,299,299,L,93.64,c1ab3528890d9adb8a7aab7115d8cd0e
COVID-3269.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3269.png,COVID/images/COVID-3269.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3269.png,COVID/masks/COVID-3269.png,True,False,299,299,L,99.18,7c6c874399ee5d7b830072fd3e32d6d9
COVID-327.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-327.png,COVID/images/COVID-327.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-327.png,COVID/masks/COVID-327.png,True,False,299,299,L,157.97,00db10b7593262c08263b119016f22bf
COVID-3270.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3270.png,COVID/images/COVID-3270.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3270.png,COVID/masks/COVID-3270.png,True,False,299,299,L,103.79,912610b1146df11e40a7ea44693a2999
COVID-3271.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3271.png,COVID/images/COVID-3271.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3271.png,COVID/masks/COVID-3271.png,True,False,299,299,L,161.92,4408a811f5cc68bdf44ab33da86c7990
COVID-3272.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3272.png,COVID/images/COVID-3272.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3272.png,COVID/masks/COVID-3272.png,True,False,299,299,L,136.24,8b96b9aa45ba2ba1c20fbc8c9ba7bb6b
COVID-3273.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3273.png,COVID/images/COVID-3273.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3273.png,COVID/masks/COVID-3273.png,True,False,299,299,L,137.01,280d8d2916258666ec4c62d33715ba78
COVID-3274.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3274.png,COVID/images/COVID-3274.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3274.png,COVID/masks/COVID-3274.png,True,False,299,299,L,118.27,c46f9aa654c35c7f355a13979df61205
COVID-3275.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3275.png,COVID/images/COVID-3275.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3275.png,COVID/masks/COVID-3275.png,True,False,299,299,L,76.98,2ac2badf314f1fa72b73c5f9ac2a3817
COVID-3276.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3276.png,COVID/images/COVID-3276.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3276.png,COVID/masks/COVID-3276.png,True,False,299,299,L,120.77,92e3cde714d35c5bedfee882a17119e0
COVID-3277.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3277.png,COVID/images/COVID-3277.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3277.png,COVID/masks/COVID-3277.png,True,False,299,299,L,141.61,3ef9ad643fbcb9d8615bc829f405801c
COVID-3278.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3278.png,COVID/images/COVID-3278.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3278.png,COVID/masks/COVID-3278.png,True,False,299,299,L,145.5,a72989506991799cfdde2bd2586c355a
COVID-3279.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3279.png,COVID/images/COVID-3279.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3279.png,COVID/masks/COVID-3279.png,True,False,299,299,L,125.49,2a0ea1133a11ca8c3f0930f356c7bc3a
COVID-328.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-328.png,COVID/images/COVID-328.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-328.png,COVID/masks/COVID-328.png,True,False,299,299,L,130.73,7bdf8a0411d30757de0b00997c9477bc
COVID-3280.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3280.png,COVID/images/COVID-3280.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3280.png,COVID/masks/COVID-3280.png,True,False,299,299,L,142.57,795ee30735f0c401d290a04869f6c5d2
COVID-3281.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3281.png,COVID/images/COVID-3281.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3281.png,COVID/masks/COVID-3281.png,True,False,299,299,L,68.58,31b30872b7710272514ad2957d352933
COVID-3282.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3282.png,COVID/images/COVID-3282.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3282.png,COVID/masks/COVID-3282.png,True,False,299,299,L,78.34,283c18374593775fa862a3753d18bcb6
COVID-3283.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3283.png,COVID/images/COVID-3283.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3283.png,COVID/masks/COVID-3283.png,True,False,299,299,L,208.46,1a2025781f64af3f3732c84ebe18e2e5
COVID-3284.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3284.png,COVID/images/COVID-3284.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3284.png,COVID/masks/COVID-3284.png,True,False,299,299,L,130.66,9614deb1bab8e6b9568f40bdde896979
COVID-3285.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3285.png,COVID/images/COVID-3285.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3285.png,COVID/masks/COVID-3285.png,True,False,299,299,L,144.35,9ae61057a52300197d38612f29eba938
COVID-3286.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3286.png,COVID/images/COVID-3286.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3286.png,COVID/masks/COVID-3286.png,True,False,299,299,L,71.32,369ae454daa8e699aea1e891318f51e5
COVID-3287.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3287.png,COVID/images/COVID-3287.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3287.png,COVID/masks/COVID-3287.png,True,False,299,299,L,138.35,11e43b740836b5afdbca2696c591159e
COVID-3288.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3288.png,COVID/images/COVID-3288.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3288.png,COVID/masks/COVID-3288.png,True,False,299,299,L,159.87,0cd0bb06120a9c5a62668219fa5ac84f
COVID-3289.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3289.png,COVID/images/COVID-3289.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3289.png,COVID/masks/COVID-3289.png,True,False,299,299,L,133.46,9c3cacf29f16c9d97f9a3a415c699872
COVID-329.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-329.png,COVID/images/COVID-329.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-329.png,COVID/masks/COVID-329.png,True,False,299,299,L,155.53,3102dbce2369bd4b38f630d7a71db311
COVID-3290.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3290.png,COVID/images/COVID-3290.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3290.png,COVID/masks/COVID-3290.png,True,False,299,299,L,185.63,24fbf1b8d1d24e55893fdfcad92b5bb2
COVID-3291.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3291.png,COVID/images/COVID-3291.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3291.png,COVID/masks/COVID-3291.png,True,False,299,299,L,144.13,dec37d2e72a2fd36ef47300b3d8a1b50
COVID-3292.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3292.png,COVID/images/COVID-3292.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3292.png,COVID/masks/COVID-3292.png,True,False,299,299,L,129.37,34948bc851d4c1254b2448b25dd5b523
COVID-3293.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3293.png,COVID/images/COVID-3293.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3293.png,COVID/masks/COVID-3293.png,True,False,299,299,L,123.78,cedb1b3d4d8748c9dd1b1aee89c67e62
COVID-3294.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3294.png,COVID/images/COVID-3294.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3294.png,COVID/masks/COVID-3294.png,True,False,299,299,L,139.01,bea520baa1dae644733fb82a015c9b83
COVID-3295.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3295.png,COVID/images/COVID-3295.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3295.png,COVID/masks/COVID-3295.png,True,False,299,299,L,129.94,98905011afe5aafff29112b8f19d9169
COVID-3296.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3296.png,COVID/images/COVID-3296.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3296.png,COVID/masks/COVID-3296.png,True,False,299,299,L,106.86,37b7bda3074d5c892095a6aaa391b96a
COVID-3297.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3297.png,COVID/images/COVID-3297.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3297.png,COVID/masks/COVID-3297.png,True,False,299,299,L,146.24,c706999e603806ceedd9e5e3dad9afba
COVID-3298.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3298.png,COVID/images/COVID-3298.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3298.png,COVID/masks/COVID-3298.png,True,False,299,299,L,132.81,13e38943c57770d1fe413482917b709c
COVID-3299.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3299.png,COVID/images/COVID-3299.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3299.png,COVID/masks/COVID-3299.png,True,False,299,299,L,132.81,13e38943c57770d1fe413482917b709c
COVID-33.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-33.png,COVID/images/COVID-33.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-33.png,COVID/masks/COVID-33.png,True,False,299,299,L,126.81,45ab3920207f014df280d839ab696524
COVID-330.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-330.png,COVID/images/COVID-330.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-330.png,COVID/masks/COVID-330.png,True,False,299,299,L,164.69,361d23d657398bf945d3010d1e96e5fc
COVID-3300.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3300.png,COVID/images/COVID-3300.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3300.png,COVID/masks/COVID-3300.png,True,False,299,299,L,127.86,56aa23dafb03c3156066f9e344cb3b00
COVID-3301.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3301.png,COVID/images/COVID-3301.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3301.png,COVID/masks/COVID-3301.png,True,False,299,299,L,92.81,14af9d92ce389edd30622f4b103787ff
COVID-3302.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3302.png,COVID/images/COVID-3302.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3302.png,COVID/masks/COVID-3302.png,True,False,299,299,L,93.04,a04ad0bdbdc92a60d018835b98a1642e
COVID-3303.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3303.png,COVID/images/COVID-3303.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3303.png,COVID/masks/COVID-3303.png,True,False,299,299,L,128.98,d934f01a639c6e104d769143769598c5
COVID-3304.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3304.png,COVID/images/COVID-3304.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3304.png,COVID/masks/COVID-3304.png,True,False,299,299,L,127.29,4b8f437adde986e259f8483726028bf3
COVID-3305.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3305.png,COVID/images/COVID-3305.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3305.png,COVID/masks/COVID-3305.png,True,False,299,299,L,180.05,c01bd5504baf1c2c348a3068ed2878c6
COVID-3306.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3306.png,COVID/images/COVID-3306.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3306.png,COVID/masks/COVID-3306.png,True,False,299,299,L,136.85,686f2eff43aff494d20fb15c996945cb
COVID-3307.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3307.png,COVID/images/COVID-3307.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3307.png,COVID/masks/COVID-3307.png,True,False,299,299,L,138.7,34be61079dd79ef0452e522e8d94b68a
COVID-3308.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3308.png,COVID/images/COVID-3308.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3308.png,COVID/masks/COVID-3308.png,True,False,299,299,L,83.92,9ede361d6d8c0c996926419765739994
COVID-3309.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3309.png,COVID/images/COVID-3309.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3309.png,COVID/masks/COVID-3309.png,True,False,299,299,L,106.63,d10fc0b58d68043a20ed814526c5f7a8
COVID-331.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-331.png,COVID/images/COVID-331.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-331.png,COVID/masks/COVID-331.png,True,False,299,299,L,156.14,d554f86cfb261e27adc51fdbf50b574c
COVID-3310.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3310.png,COVID/images/COVID-3310.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3310.png,COVID/masks/COVID-3310.png,True,False,299,299,L,135.61,3c8b3a41ad15e72a67224a642c5cc251
COVID-3311.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3311.png,COVID/images/COVID-3311.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3311.png,COVID/masks/COVID-3311.png,True,False,299,299,L,157.91,68397fb4aea5bc62f15f6a0379dfbdfe
COVID-3312.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3312.png,COVID/images/COVID-3312.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3312.png,COVID/masks/COVID-3312.png,True,False,299,299,L,160.75,207af72522dca676f32c4dbf21e10cb5
COVID-3313.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3313.png,COVID/images/COVID-3313.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3313.png,COVID/masks/COVID-3313.png,True,False,299,299,L,165.14,4ba1b88dd0544e278e89b3bf4bb5248a
COVID-3314.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3314.png,COVID/images/COVID-3314.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3314.png,COVID/masks/COVID-3314.png,True,False,299,299,L,139.72,10e27c8afae29437ee9358d9f2f60f9f
COVID-3315.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3315.png,COVID/images/COVID-3315.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3315.png,COVID/masks/COVID-3315.png,True,False,299,299,L,105.17,6babdbf1263c64d17c5354f3bb5fe6b4
COVID-3316.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3316.png,COVID/images/COVID-3316.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3316.png,COVID/masks/COVID-3316.png,True,False,299,299,L,110.74,519d55a8da2675b5cddd83cadc997573
COVID-3317.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3317.png,COVID/images/COVID-3317.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3317.png,COVID/masks/COVID-3317.png,True,False,299,299,L,74.84,2ea42e96963aa579cddc17a2217513a4
COVID-3318.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3318.png,COVID/images/COVID-3318.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3318.png,COVID/masks/COVID-3318.png,True,False,299,299,L,142.97,ff12b0f4ff2223b3f031a2f95f07f036
COVID-3319.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3319.png,COVID/images/COVID-3319.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3319.png,COVID/masks/COVID-3319.png,True,False,299,299,L,160.08,50ceb3b5586a736d11b16ac4318d50a2
COVID-332.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-332.png,COVID/images/COVID-332.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-332.png,COVID/masks/COVID-332.png,True,False,299,299,L,134.35,7aa88e6484deecb6d27a6b5ee51b76db
COVID-3320.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3320.png,COVID/images/COVID-3320.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3320.png,COVID/masks/COVID-3320.png,True,False,299,299,L,124.37,a2e2942cc45720ca99e00b9a53e2acf3
COVID-3321.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3321.png,COVID/images/COVID-3321.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3321.png,COVID/masks/COVID-3321.png,True,False,299,299,L,138.81,66b4a326b353d4dce0cf538d9bccc028
COVID-3322.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3322.png,COVID/images/COVID-3322.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3322.png,COVID/masks/COVID-3322.png,True,False,299,299,L,113.5,b7bb78bb828c50994e143c068aaa9845
COVID-3323.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3323.png,COVID/images/COVID-3323.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3323.png,COVID/masks/COVID-3323.png,True,False,299,299,L,96.71,769d325c842424e043b75d766ea7f0fc
COVID-3324.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3324.png,COVID/images/COVID-3324.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3324.png,COVID/masks/COVID-3324.png,True,False,299,299,L,131.38,2aed22215a63b6fe77d31a0d5e850c9e
COVID-3325.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3325.png,COVID/images/COVID-3325.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3325.png,COVID/masks/COVID-3325.png,True,False,299,299,L,112.85,d9d530e850b594c95efda3b220e11b8a
COVID-3326.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3326.png,COVID/images/COVID-3326.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3326.png,COVID/masks/COVID-3326.png,True,False,299,299,L,79.58,993b58b7ed2d435ffe48246f3434d087
COVID-3327.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3327.png,COVID/images/COVID-3327.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3327.png,COVID/masks/COVID-3327.png,True,False,299,299,L,174.56,b86568c4028bb74411251c360a07bf87
COVID-3328.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3328.png,COVID/images/COVID-3328.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3328.png,COVID/masks/COVID-3328.png,True,False,299,299,L,137.99,c4115133660a1f9aac99c3b98d2b58fe
COVID-3329.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3329.png,COVID/images/COVID-3329.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3329.png,COVID/masks/COVID-3329.png,True,False,299,299,L,95.66,12dbe39f0cd4b28291d1284cabcad3f2
COVID-333.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-333.png,COVID/images/COVID-333.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-333.png,COVID/masks/COVID-333.png,True,False,299,299,L,134.96,13db5dd53e527dd73e0dca8e95177813
COVID-3330.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3330.png,COVID/images/COVID-3330.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3330.png,COVID/masks/COVID-3330.png,True,False,299,299,L,89.11,696ca49f20ed9260475c107bef047016
COVID-3331.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3331.png,COVID/images/COVID-3331.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3331.png,COVID/masks/COVID-3331.png,True,False,299,299,L,143.85,c9b65717265532f248c0fb2e70643936
COVID-3332.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3332.png,COVID/images/COVID-3332.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3332.png,COVID/masks/COVID-3332.png,True,False,299,299,L,117.33,1e8efbe5522905386413289619ebddcd
COVID-3333.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3333.png,COVID/images/COVID-3333.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3333.png,COVID/masks/COVID-3333.png,True,False,299,299,L,138.88,74d34da366f48a12f9b0b14e128fe6ba
COVID-3334.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3334.png,COVID/images/COVID-3334.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3334.png,COVID/masks/COVID-3334.png,True,False,299,299,L,125.22,459a68c76e0826aa06be778e362f9b1e
COVID-3335.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3335.png,COVID/images/COVID-3335.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3335.png,COVID/masks/COVID-3335.png,True,False,299,299,L,111.77,0b094ecfc1b1810b2654080d5491d8a7
COVID-3336.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3336.png,COVID/images/COVID-3336.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3336.png,COVID/masks/COVID-3336.png,True,False,299,299,L,125.97,4d1596154f37420f286f3a950a47ecd8
COVID-3337.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3337.png,COVID/images/COVID-3337.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3337.png,COVID/masks/COVID-3337.png,True,False,299,299,L,123.45,43ff313ea3adbe3f89bc7d95d8bee839
COVID-3338.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3338.png,COVID/images/COVID-3338.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3338.png,COVID/masks/COVID-3338.png,True,False,299,299,L,130.68,57cd06de67546738862a41acd4fa1b66
COVID-3339.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3339.png,COVID/images/COVID-3339.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3339.png,COVID/masks/COVID-3339.png,True,False,299,299,L,99.93,fb1f86af77ca042bad8ce5c3833793ab
COVID-334.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-334.png,COVID/images/COVID-334.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-334.png,COVID/masks/COVID-334.png,True,False,299,299,L,153.52,89a37083045f12683ecb995c3b32c3e9
COVID-3340.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3340.png,COVID/images/COVID-3340.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3340.png,COVID/masks/COVID-3340.png,True,False,299,299,L,104.3,515b1ddc8eb9a48ad5402fc10004b7d5
COVID-3341.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3341.png,COVID/images/COVID-3341.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3341.png,COVID/masks/COVID-3341.png,True,False,299,299,L,142.97,dae2b704c8f119835e11439d3b71ff59
COVID-3342.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3342.png,COVID/images/COVID-3342.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3342.png,COVID/masks/COVID-3342.png,True,False,299,299,L,140.37,bc8cf9e31e39f3695a54d9fbd02ac2ef
COVID-3343.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3343.png,COVID/images/COVID-3343.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3343.png,COVID/masks/COVID-3343.png,True,False,299,299,L,151.88,84d5ef09cd058f335a07843403aab9fb
COVID-3344.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3344.png,COVID/images/COVID-3344.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3344.png,COVID/masks/COVID-3344.png,True,False,299,299,L,93.59,c83596b2f264762f5158ad3857f1fcc3
COVID-3345.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3345.png,COVID/images/COVID-3345.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3345.png,COVID/masks/COVID-3345.png,True,False,299,299,L,135.36,3d4fab934bb1bf6dfc7a0689c24af4af
COVID-3346.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3346.png,COVID/images/COVID-3346.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3346.png,COVID/masks/COVID-3346.png,True,False,299,299,L,136.75,e0acd4754a0f1bd8e2cadb7e657ed507
COVID-3347.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3347.png,COVID/images/COVID-3347.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3347.png,COVID/masks/COVID-3347.png,True,False,299,299,L,136.72,645a9b9e5529ee0e6387b1e1b8c5a43f
COVID-3348.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3348.png,COVID/images/COVID-3348.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3348.png,COVID/masks/COVID-3348.png,True,False,299,299,L,141.24,7e2cff7deaa0a30bb0462559c54fd866
COVID-3349.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3349.png,COVID/images/COVID-3349.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3349.png,COVID/masks/COVID-3349.png,True,False,299,299,L,159.62,37da86d2a43ce38fa5ac276bc815bddd
COVID-335.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-335.png,COVID/images/COVID-335.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-335.png,COVID/masks/COVID-335.png,True,False,299,299,L,141.43,458fa26406805133d5395e5237f31459
COVID-3350.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3350.png,COVID/images/COVID-3350.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3350.png,COVID/masks/COVID-3350.png,True,False,299,299,L,166.59,161be832e9b408030f8ca567f46647bd
COVID-3351.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3351.png,COVID/images/COVID-3351.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3351.png,COVID/masks/COVID-3351.png,True,False,299,299,L,133.3,c5e54614be97a935f1b4ee59865be53b
COVID-3352.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3352.png,COVID/images/COVID-3352.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3352.png,COVID/masks/COVID-3352.png,True,False,299,299,L,117.89,5a9021fb6fc4c222b6f5990979ebb653
COVID-3353.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3353.png,COVID/images/COVID-3353.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3353.png,COVID/masks/COVID-3353.png,True,False,299,299,L,96.88,a71a489036b84a67411b7fc5dc019331
COVID-3354.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3354.png,COVID/images/COVID-3354.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3354.png,COVID/masks/COVID-3354.png,True,False,299,299,L,138.11,1990179175889353fae2711acc11918a
COVID-3355.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3355.png,COVID/images/COVID-3355.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3355.png,COVID/masks/COVID-3355.png,True,False,299,299,L,97.42,811bdf0b63416131673eed6b5a5d3a14
COVID-3356.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3356.png,COVID/images/COVID-3356.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3356.png,COVID/masks/COVID-3356.png,True,False,299,299,L,165.29,8f4f3ff885a0ce2485e9ac2c6dc98e54
COVID-3357.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3357.png,COVID/images/COVID-3357.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3357.png,COVID/masks/COVID-3357.png,True,False,299,299,L,91.24,20a27aee7d6e401f042dc805aa5370a2
COVID-3358.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3358.png,COVID/images/COVID-3358.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3358.png,COVID/masks/COVID-3358.png,True,False,299,299,L,71.95,4bfb969e85ff5a817532e43d41057c0a
COVID-3359.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3359.png,COVID/images/COVID-3359.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3359.png,COVID/masks/COVID-3359.png,True,False,299,299,L,97.83,9102d7f39c7631469606bc169270dcad
COVID-336.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-336.png,COVID/images/COVID-336.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-336.png,COVID/masks/COVID-336.png,True,False,299,299,L,162.73,1419daf6f690badb7d55bc8b7bab5370
COVID-3360.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3360.png,COVID/images/COVID-3360.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3360.png,COVID/masks/COVID-3360.png,True,False,299,299,L,102.22,d1ef2eeb28cdac27218bca5af217760b
COVID-3361.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3361.png,COVID/images/COVID-3361.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3361.png,COVID/masks/COVID-3361.png,True,False,299,299,L,105.71,8819dca5853ca3bb05640405243e3112
COVID-3362.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3362.png,COVID/images/COVID-3362.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3362.png,COVID/masks/COVID-3362.png,True,False,299,299,L,115.61,e819ddd2671de293cb11f61f1f302e4e
COVID-3363.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3363.png,COVID/images/COVID-3363.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3363.png,COVID/masks/COVID-3363.png,True,False,299,299,L,86.19,426dd3410c28e8bc205712bed8873953
COVID-3364.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3364.png,COVID/images/COVID-3364.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3364.png,COVID/masks/COVID-3364.png,True,False,299,299,L,134.87,c2dfe20aa9a71ba246dd26ab14e32fc6
COVID-3365.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3365.png,COVID/images/COVID-3365.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3365.png,COVID/masks/COVID-3365.png,True,False,299,299,L,191.46,3ef26212ef2e8745bcf25b9c8affd194
COVID-3366.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3366.png,COVID/images/COVID-3366.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3366.png,COVID/masks/COVID-3366.png,True,False,299,299,L,82.44,9dee09b7aeb1519317ce11d5742520e9
COVID-3367.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3367.png,COVID/images/COVID-3367.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3367.png,COVID/masks/COVID-3367.png,True,False,299,299,L,126.84,82023fbfed427366dadc148c1dbd90f6
COVID-3368.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3368.png,COVID/images/COVID-3368.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3368.png,COVID/masks/COVID-3368.png,True,False,299,299,L,133.1,4f71d780527c76a343da080ef7faf149
COVID-3369.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3369.png,COVID/images/COVID-3369.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3369.png,COVID/masks/COVID-3369.png,True,False,299,299,L,101.9,9589b1bc07328cf50dabe48f88f2c781
COVID-337.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-337.png,COVID/images/COVID-337.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-337.png,COVID/masks/COVID-337.png,True,False,299,299,L,136.82,6459110d270ec5b61bb3d97b771dd89b
COVID-3370.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3370.png,COVID/images/COVID-3370.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3370.png,COVID/masks/COVID-3370.png,True,False,299,299,L,135.2,ac92e3f4c5c1ec8cb6bc41fe407d1fc2
COVID-3371.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3371.png,COVID/images/COVID-3371.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3371.png,COVID/masks/COVID-3371.png,True,False,299,299,L,109.15,1ad7025fe285f4f7fa4ea494842dafc2
COVID-3372.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3372.png,COVID/images/COVID-3372.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3372.png,COVID/masks/COVID-3372.png,True,False,299,299,L,116.94,3177ffdd6462104fb0a7fca12e2c41ea
COVID-3373.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3373.png,COVID/images/COVID-3373.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3373.png,COVID/masks/COVID-3373.png,True,False,299,299,L,139.02,c5438ddcb2ef4d16bd0dae1b8e927b06
COVID-3374.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3374.png,COVID/images/COVID-3374.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3374.png,COVID/masks/COVID-3374.png,True,False,299,299,L,139.02,c5438ddcb2ef4d16bd0dae1b8e927b06
COVID-3375.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3375.png,COVID/images/COVID-3375.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3375.png,COVID/masks/COVID-3375.png,True,False,299,299,L,139.02,c5438ddcb2ef4d16bd0dae1b8e927b06
COVID-3376.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3376.png,COVID/images/COVID-3376.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3376.png,COVID/masks/COVID-3376.png,True,False,299,299,L,139.02,c5438ddcb2ef4d16bd0dae1b8e927b06
COVID-3377.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3377.png,COVID/images/COVID-3377.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3377.png,COVID/masks/COVID-3377.png,True,False,299,299,L,139.4,856a3ad172c183b3c18fc3137ca368e9
COVID-3378.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3378.png,COVID/images/COVID-3378.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3378.png,COVID/masks/COVID-3378.png,True,False,299,299,L,167.34,cb1fa78de4303d4f1c650f98de23290e
COVID-3379.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3379.png,COVID/images/COVID-3379.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3379.png,COVID/masks/COVID-3379.png,True,False,299,299,L,115.97,f8fdec133e33338e962720adf18fc9ef
COVID-338.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-338.png,COVID/images/COVID-338.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-338.png,COVID/masks/COVID-338.png,True,False,299,299,L,156.38,5794067284d6f9dd93e5e8d52e90e803
COVID-3380.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3380.png,COVID/images/COVID-3380.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3380.png,COVID/masks/COVID-3380.png,True,False,299,299,L,137.07,1e057c0f63c57d19e4c37eff32b85b4b
COVID-3381.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3381.png,COVID/images/COVID-3381.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3381.png,COVID/masks/COVID-3381.png,True,False,299,299,L,112.36,299c236e07df0824185cda9ae6087011
COVID-3382.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3382.png,COVID/images/COVID-3382.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3382.png,COVID/masks/COVID-3382.png,True,False,299,299,L,109.96,e777262a1093267eec4fdfb742bd74af
COVID-3383.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3383.png,COVID/images/COVID-3383.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3383.png,COVID/masks/COVID-3383.png,True,False,299,299,L,88.38,b54ae6da7a3d3799fff3411ab27ffc82
COVID-3384.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3384.png,COVID/images/COVID-3384.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3384.png,COVID/masks/COVID-3384.png,True,False,299,299,L,130.8,a711190521828801e5dd002d1f636615
COVID-3385.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3385.png,COVID/images/COVID-3385.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3385.png,COVID/masks/COVID-3385.png,True,False,299,299,L,129.29,1680582ad9b67a40d7fd9465fd3065e0
COVID-3386.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3386.png,COVID/images/COVID-3386.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3386.png,COVID/masks/COVID-3386.png,True,False,299,299,L,142.98,04ba1429bfcbdcfdd2759b7402e4c107
COVID-3387.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3387.png,COVID/images/COVID-3387.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3387.png,COVID/masks/COVID-3387.png,True,False,299,299,L,130.6,75bdbda7ab780bc2e1ca3b988591956a
COVID-3388.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3388.png,COVID/images/COVID-3388.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3388.png,COVID/masks/COVID-3388.png,True,False,299,299,L,139.35,db8b6328cadd052ebd33c6d7128ea0f1
COVID-3389.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3389.png,COVID/images/COVID-3389.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3389.png,COVID/masks/COVID-3389.png,True,False,299,299,L,142.25,60093a10397087cf0109440c593659a7
COVID-339.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-339.png,COVID/images/COVID-339.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-339.png,COVID/masks/COVID-339.png,True,False,299,299,L,160.55,a4d54b44691865b8ab80b820c820c5e7
COVID-3390.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3390.png,COVID/images/COVID-3390.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3390.png,COVID/masks/COVID-3390.png,True,False,299,299,L,124.78,62b44cf85a19542612bc0cb0f22c2c55
COVID-3391.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3391.png,COVID/images/COVID-3391.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3391.png,COVID/masks/COVID-3391.png,True,False,299,299,L,98.88,2fc9e93d0d6866f27cc75b9cf51b20ad
COVID-3392.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3392.png,COVID/images/COVID-3392.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3392.png,COVID/masks/COVID-3392.png,True,False,299,299,L,119.67,c2be22202d279d52032ee051ee6e0e91
COVID-3393.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3393.png,COVID/images/COVID-3393.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3393.png,COVID/masks/COVID-3393.png,True,False,299,299,L,139.85,bd3a440cd674130a203bf7badcbf03e9
COVID-3394.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3394.png,COVID/images/COVID-3394.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3394.png,COVID/masks/COVID-3394.png,True,False,299,299,L,197.92,de7471b328ebda74c05dfbc8d673c45e
COVID-3395.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3395.png,COVID/images/COVID-3395.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3395.png,COVID/masks/COVID-3395.png,True,False,299,299,L,115.56,67a5cf0a2de8844c27226370ba57fea7
COVID-3396.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3396.png,COVID/images/COVID-3396.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3396.png,COVID/masks/COVID-3396.png,True,False,299,299,L,77.42,4ea9b9a82779a5b22e1d6d45b69fc227
COVID-3397.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3397.png,COVID/images/COVID-3397.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3397.png,COVID/masks/COVID-3397.png,True,False,299,299,L,139.73,a84e3ad511ec92dac9591c065a4ac930
COVID-3398.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3398.png,COVID/images/COVID-3398.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3398.png,COVID/masks/COVID-3398.png,True,False,299,299,L,135.03,ab96c909cd24570355fe96355ced3bfb
COVID-3399.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3399.png,COVID/images/COVID-3399.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3399.png,COVID/masks/COVID-3399.png,True,False,299,299,L,91.14,247fd1dc33a3cf000908d2df3effd31c
COVID-34.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-34.png,COVID/images/COVID-34.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-34.png,COVID/masks/COVID-34.png,True,False,299,299,L,159.8,f7be0534e136b944752db324062133d4
COVID-340.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-340.png,COVID/images/COVID-340.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-340.png,COVID/masks/COVID-340.png,True,False,299,299,L,110.46,7ffc478153d0540f5e5076f6dedf12e9
COVID-3400.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3400.png,COVID/images/COVID-3400.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3400.png,COVID/masks/COVID-3400.png,True,False,299,299,L,105.88,c1ce70eafebd605a93d2e81c8eff11ff
COVID-3401.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3401.png,COVID/images/COVID-3401.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3401.png,COVID/masks/COVID-3401.png,True,False,299,299,L,113.46,d45cb0ef4dece2da3cdec2127a5ece3a
COVID-3402.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3402.png,COVID/images/COVID-3402.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3402.png,COVID/masks/COVID-3402.png,True,False,299,299,L,131.43,9dcabb580b6198a675f8860c62a436e0
COVID-3403.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3403.png,COVID/images/COVID-3403.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3403.png,COVID/masks/COVID-3403.png,True,False,299,299,L,131.43,9dcabb580b6198a675f8860c62a436e0
COVID-3404.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3404.png,COVID/images/COVID-3404.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3404.png,COVID/masks/COVID-3404.png,True,False,299,299,L,160.59,c2a4c09e348046dae920c43e19715e09
COVID-3405.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3405.png,COVID/images/COVID-3405.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3405.png,COVID/masks/COVID-3405.png,True,False,299,299,L,133.25,c9c8a98d23cc6707a40cdbd1d317c9d4
COVID-3406.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3406.png,COVID/images/COVID-3406.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3406.png,COVID/masks/COVID-3406.png,True,False,299,299,L,132.25,e4c0a1ec16cb05b8bb2a654b7461f440
COVID-3407.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3407.png,COVID/images/COVID-3407.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3407.png,COVID/masks/COVID-3407.png,True,False,299,299,L,101.23,c81aa23c444374b070230963582648e4
COVID-3408.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3408.png,COVID/images/COVID-3408.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3408.png,COVID/masks/COVID-3408.png,True,False,299,299,L,112.61,016526304ce9d03dbadd786f4ce964c3
COVID-3409.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3409.png,COVID/images/COVID-3409.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3409.png,COVID/masks/COVID-3409.png,True,False,299,299,L,134.88,5a0efcacdf759ca9de7942dc5fa22661
COVID-341.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-341.png,COVID/images/COVID-341.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-341.png,COVID/masks/COVID-341.png,True,False,299,299,L,157.99,ee921088d691b311cd3b1bee357f6aa3
COVID-3410.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3410.png,COVID/images/COVID-3410.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3410.png,COVID/masks/COVID-3410.png,True,False,299,299,L,95.92,efe76f02bc83d87ebb9e3df3ca2c789c
COVID-3411.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3411.png,COVID/images/COVID-3411.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3411.png,COVID/masks/COVID-3411.png,True,False,299,299,L,144.69,6bc31323d96a09328d5a7bc561e20ad5
COVID-3412.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3412.png,COVID/images/COVID-3412.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3412.png,COVID/masks/COVID-3412.png,True,False,299,299,L,136.65,bd332b0aba30cbef61d75a44db2648ca
COVID-3413.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3413.png,COVID/images/COVID-3413.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3413.png,COVID/masks/COVID-3413.png,True,False,299,299,L,84.85,95285c05628f3a40fddfb72770ebe001
COVID-3414.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3414.png,COVID/images/COVID-3414.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3414.png,COVID/masks/COVID-3414.png,True,False,299,299,L,158.09,b1bc22e3e5f9552496188781a9dbcdb6
COVID-3415.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3415.png,COVID/images/COVID-3415.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3415.png,COVID/masks/COVID-3415.png,True,False,299,299,L,140.3,f1294b62458e8674942192285dd96758
COVID-3416.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3416.png,COVID/images/COVID-3416.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3416.png,COVID/masks/COVID-3416.png,True,False,299,299,L,119.14,626808e5a2ba4057e4dc8782cb731bce
COVID-3417.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3417.png,COVID/images/COVID-3417.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3417.png,COVID/masks/COVID-3417.png,True,False,299,299,L,108.67,2043382e84f7e89301d5f290aad053fc
COVID-3418.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3418.png,COVID/images/COVID-3418.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3418.png,COVID/masks/COVID-3418.png,True,False,299,299,L,153.62,13f59d7af322795935a4a41d4df6b6a8
COVID-3419.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3419.png,COVID/images/COVID-3419.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3419.png,COVID/masks/COVID-3419.png,True,False,299,299,L,156.76,d05c06aca9d7742fb3054a8b4cbc026f
COVID-342.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-342.png,COVID/images/COVID-342.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-342.png,COVID/masks/COVID-342.png,True,False,299,299,L,159.12,0f081094e8d02b8fcb46941b1d3f03ae
COVID-3420.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3420.png,COVID/images/COVID-3420.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3420.png,COVID/masks/COVID-3420.png,True,False,299,299,L,148.17,6e66cd40f556700e85fd791f06a4a54f
COVID-3421.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3421.png,COVID/images/COVID-3421.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3421.png,COVID/masks/COVID-3421.png,True,False,299,299,L,111.33,ece7c27761ceaebb34a6c26674da5e3a
COVID-3422.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3422.png,COVID/images/COVID-3422.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3422.png,COVID/masks/COVID-3422.png,True,False,299,299,L,102.1,c3ad7662163b7e631b3097c4f184f405
COVID-3423.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3423.png,COVID/images/COVID-3423.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3423.png,COVID/masks/COVID-3423.png,True,False,299,299,L,165.36,00452e4bb8679373672c0d8d4179b4ab
COVID-3424.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3424.png,COVID/images/COVID-3424.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3424.png,COVID/masks/COVID-3424.png,True,False,299,299,L,152.0,96a85d40c020052af95dd7cde811d5a1
COVID-3425.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3425.png,COVID/images/COVID-3425.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3425.png,COVID/masks/COVID-3425.png,True,False,299,299,L,173.22,34de7411cf1973c29ea3e30e0a0bdf49
COVID-3426.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3426.png,COVID/images/COVID-3426.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3426.png,COVID/masks/COVID-3426.png,True,False,299,299,L,133.79,c3fc7ea61ad8460554276112852d7375
COVID-3427.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3427.png,COVID/images/COVID-3427.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3427.png,COVID/masks/COVID-3427.png,True,False,299,299,L,133.79,c3fc7ea61ad8460554276112852d7375
COVID-3428.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3428.png,COVID/images/COVID-3428.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3428.png,COVID/masks/COVID-3428.png,True,False,299,299,L,140.28,958760da197ae6522df0afac5c3a1d08
COVID-3429.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3429.png,COVID/images/COVID-3429.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3429.png,COVID/masks/COVID-3429.png,True,False,299,299,L,140.28,958760da197ae6522df0afac5c3a1d08
COVID-343.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-343.png,COVID/images/COVID-343.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-343.png,COVID/masks/COVID-343.png,True,False,299,299,L,174.11,c158865eee627db98d6b2b4a6b890291
COVID-3430.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3430.png,COVID/images/COVID-3430.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3430.png,COVID/masks/COVID-3430.png,True,False,299,299,L,138.45,eb011daddc39f02758854d4fc2b926f6
COVID-3431.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3431.png,COVID/images/COVID-3431.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3431.png,COVID/masks/COVID-3431.png,True,False,299,299,L,155.14,396b1521bfa1571e884cfb086e2ecd1a
COVID-3432.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3432.png,COVID/images/COVID-3432.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3432.png,COVID/masks/COVID-3432.png,True,False,299,299,L,139.89,1eb05b2d5396197a5a25043228a7c6ef
COVID-3433.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3433.png,COVID/images/COVID-3433.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3433.png,COVID/masks/COVID-3433.png,True,False,299,299,L,83.31,c647463afa3a5a33f00dbdac5e69808d
COVID-3434.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3434.png,COVID/images/COVID-3434.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3434.png,COVID/masks/COVID-3434.png,True,False,299,299,L,131.31,175ee934a1a5a2e74dffe30b192d296d
COVID-3435.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3435.png,COVID/images/COVID-3435.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3435.png,COVID/masks/COVID-3435.png,True,False,299,299,L,144.42,1f39e3cb9a7add4e98554009e90a53a4
COVID-3436.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3436.png,COVID/images/COVID-3436.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3436.png,COVID/masks/COVID-3436.png,True,False,299,299,L,113.36,2c0319485e9bec70a04f18f9b4813a12
COVID-3437.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3437.png,COVID/images/COVID-3437.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3437.png,COVID/masks/COVID-3437.png,True,False,299,299,L,79.98,b0907c72f6879e5616eb4f42ca23fb32
COVID-3438.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3438.png,COVID/images/COVID-3438.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3438.png,COVID/masks/COVID-3438.png,True,False,299,299,L,158.83,b319016916aaa8961f5fa7da261867a1
COVID-3439.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3439.png,COVID/images/COVID-3439.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3439.png,COVID/masks/COVID-3439.png,True,False,299,299,L,123.99,557b4270e8aa943aa68a515bccaafebe
COVID-344.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-344.png,COVID/images/COVID-344.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-344.png,COVID/masks/COVID-344.png,True,False,299,299,L,156.41,88e568cf993d9df541ef4bae362956cf
COVID-3440.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3440.png,COVID/images/COVID-3440.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3440.png,COVID/masks/COVID-3440.png,True,False,299,299,L,178.5,f9e6c4bca51915cd12e432c19c911b3a
COVID-3441.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3441.png,COVID/images/COVID-3441.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3441.png,COVID/masks/COVID-3441.png,True,False,299,299,L,148.49,7f45d6ba86fd5183890ffaa2bca6d78f
COVID-3442.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3442.png,COVID/images/COVID-3442.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3442.png,COVID/masks/COVID-3442.png,True,False,299,299,L,173.78,7d7fc4302a553452447d9280f669a4dc
COVID-3443.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3443.png,COVID/images/COVID-3443.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3443.png,COVID/masks/COVID-3443.png,True,False,299,299,L,153.33,cf93f99a7ce1c9df439d53d40b7ef112
COVID-3444.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3444.png,COVID/images/COVID-3444.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3444.png,COVID/masks/COVID-3444.png,True,False,299,299,L,93.11,b975f0bd75070355430b7c28127df2bb
COVID-3445.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3445.png,COVID/images/COVID-3445.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3445.png,COVID/masks/COVID-3445.png,True,False,299,299,L,104.38,e72b56859e42a49205af66c4f26fe9a0
COVID-3446.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3446.png,COVID/images/COVID-3446.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3446.png,COVID/masks/COVID-3446.png,True,False,299,299,L,154.17,992921a92116900779f70a798aef2c89
COVID-3447.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3447.png,COVID/images/COVID-3447.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3447.png,COVID/masks/COVID-3447.png,True,False,299,299,L,109.83,5b865bbd8c560f8ed77d7e8efaec503a
COVID-3448.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3448.png,COVID/images/COVID-3448.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3448.png,COVID/masks/COVID-3448.png,True,False,299,299,L,185.22,e519f9ba65b4196b9adc137561556158
COVID-3449.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3449.png,COVID/images/COVID-3449.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3449.png,COVID/masks/COVID-3449.png,True,False,299,299,L,116.06,8e19e82a23ad852a047b20c5e0e47526
COVID-345.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-345.png,COVID/images/COVID-345.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-345.png,COVID/masks/COVID-345.png,True,False,299,299,L,141.49,a8cf1666d0e172985be71604d7f9845d
COVID-3450.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3450.png,COVID/images/COVID-3450.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3450.png,COVID/masks/COVID-3450.png,True,False,299,299,L,159.27,f8cc8a887f07d7045ad91e7df6e6e2c1
COVID-3451.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3451.png,COVID/images/COVID-3451.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3451.png,COVID/masks/COVID-3451.png,True,False,299,299,L,149.24,0b4fd19ae233761d1a2002951bb8aab4
COVID-3452.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3452.png,COVID/images/COVID-3452.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3452.png,COVID/masks/COVID-3452.png,True,False,299,299,L,93.91,c3c476520d768c77d8f7046e0419a3f6
COVID-3453.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3453.png,COVID/images/COVID-3453.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3453.png,COVID/masks/COVID-3453.png,True,False,299,299,L,173.66,66d9a90d19c9b038b97a84e79215b188
COVID-3454.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3454.png,COVID/images/COVID-3454.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3454.png,COVID/masks/COVID-3454.png,True,False,299,299,L,153.54,a34851655a7e24fbaf46f5206942088e
COVID-3455.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3455.png,COVID/images/COVID-3455.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3455.png,COVID/masks/COVID-3455.png,True,False,299,299,L,129.92,8b366068ae631a8865e07917fb9d85f0
COVID-3456.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3456.png,COVID/images/COVID-3456.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3456.png,COVID/masks/COVID-3456.png,True,False,299,299,L,143.57,f142b12c9ab97a3101842f57a0af931a
COVID-3457.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3457.png,COVID/images/COVID-3457.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3457.png,COVID/masks/COVID-3457.png,True,False,299,299,L,86.43,5081e4b9403adc1d84b85d8db8f8d38e
COVID-3458.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3458.png,COVID/images/COVID-3458.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3458.png,COVID/masks/COVID-3458.png,True,False,299,299,L,126.7,2d1070ddc4bbb21d8a540df6196adf8c
COVID-3459.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3459.png,COVID/images/COVID-3459.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3459.png,COVID/masks/COVID-3459.png,True,False,299,299,L,70.78,09c0310e494284820c74e77385e8d6eb
COVID-346.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-346.png,COVID/images/COVID-346.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-346.png,COVID/masks/COVID-346.png,True,False,299,299,L,167.95,69d54ee7f48dda8a071f006cf90e1077
COVID-3460.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3460.png,COVID/images/COVID-3460.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3460.png,COVID/masks/COVID-3460.png,True,False,299,299,L,99.94,18e258d14767bae6877558cdc012ffa8
COVID-3461.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3461.png,COVID/images/COVID-3461.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3461.png,COVID/masks/COVID-3461.png,True,False,299,299,L,92.65,cc4992c4c930447d2763828b871e9969
COVID-3462.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3462.png,COVID/images/COVID-3462.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3462.png,COVID/masks/COVID-3462.png,True,False,299,299,L,71.66,8e607323b1be2d38494d6e4acc4a97d6
COVID-3463.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3463.png,COVID/images/COVID-3463.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3463.png,COVID/masks/COVID-3463.png,True,False,299,299,L,118.21,c2b891affa43cdb3e83d5d3b41696a71
COVID-3464.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3464.png,COVID/images/COVID-3464.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3464.png,COVID/masks/COVID-3464.png,True,False,299,299,L,70.05,dbefbf7a3ad3a68a213f97aff42bc607
COVID-3465.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3465.png,COVID/images/COVID-3465.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3465.png,COVID/masks/COVID-3465.png,True,False,299,299,L,87.6,588e7b1cc0d1dd99f251a57140b9e1c0
COVID-3466.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3466.png,COVID/images/COVID-3466.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3466.png,COVID/masks/COVID-3466.png,True,False,299,299,L,170.43,6e4d79f95fa2be3ce9c7005d64c299ee
COVID-3467.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3467.png,COVID/images/COVID-3467.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3467.png,COVID/masks/COVID-3467.png,True,False,299,299,L,143.61,4ab97354f08d75a745fdf4d40f38acb6
COVID-3468.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3468.png,COVID/images/COVID-3468.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3468.png,COVID/masks/COVID-3468.png,True,False,299,299,L,114.33,300df7ed4e61929d3e343da6d7ad5f16
COVID-3469.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3469.png,COVID/images/COVID-3469.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3469.png,COVID/masks/COVID-3469.png,True,False,299,299,L,158.08,e5bfb83b9671b25986a03d3414a82de9
COVID-347.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-347.png,COVID/images/COVID-347.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-347.png,COVID/masks/COVID-347.png,True,False,299,299,L,128.2,c78a8ddc7bc4b13b18c26ac55d9ec9ae
COVID-3470.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3470.png,COVID/images/COVID-3470.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3470.png,COVID/masks/COVID-3470.png,True,False,299,299,L,109.96,d6ec28c12739d40d9117310842667346
COVID-3471.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3471.png,COVID/images/COVID-3471.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3471.png,COVID/masks/COVID-3471.png,True,False,299,299,L,156.32,84284479effd5c5d0d0257b0f161d01d
COVID-3472.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3472.png,COVID/images/COVID-3472.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3472.png,COVID/masks/COVID-3472.png,True,False,299,299,L,126.98,4972663328f7192f347cd8de871f3e68
COVID-3473.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3473.png,COVID/images/COVID-3473.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3473.png,COVID/masks/COVID-3473.png,True,False,299,299,L,75.59,f11c7738477aed0b39ddd9526a49687d
COVID-3474.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3474.png,COVID/images/COVID-3474.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3474.png,COVID/masks/COVID-3474.png,True,False,299,299,L,135.38,1fc4628f37c8caaf20b01fa9c6f8ed1a
COVID-3475.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3475.png,COVID/images/COVID-3475.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3475.png,COVID/masks/COVID-3475.png,True,False,299,299,L,94.57,8d35d7cf94f30369fa82225825b381af
COVID-3476.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3476.png,COVID/images/COVID-3476.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3476.png,COVID/masks/COVID-3476.png,True,False,299,299,L,156.12,68cc2eb7b44768913654c5cc24a3a8d4
COVID-3477.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3477.png,COVID/images/COVID-3477.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3477.png,COVID/masks/COVID-3477.png,True,False,299,299,L,112.96,0614b7ce455e45accec35320c9c710ff
COVID-3478.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3478.png,COVID/images/COVID-3478.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3478.png,COVID/masks/COVID-3478.png,True,False,299,299,L,123.36,67f4f6b6e7f6d9d1f190572ae9701245
COVID-3479.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3479.png,COVID/images/COVID-3479.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3479.png,COVID/masks/COVID-3479.png,True,False,299,299,L,144.43,9bf35963f82af59fae23f77f132a8b40
COVID-348.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-348.png,COVID/images/COVID-348.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-348.png,COVID/masks/COVID-348.png,True,False,299,299,L,137.5,ff67f31a678d1a9df67ff2d52d517d9c
COVID-3480.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3480.png,COVID/images/COVID-3480.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3480.png,COVID/masks/COVID-3480.png,True,False,299,299,L,167.71,233f6c0224304ce4d43bc3a811bd8d5c
COVID-3481.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3481.png,COVID/images/COVID-3481.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3481.png,COVID/masks/COVID-3481.png,True,False,299,299,L,113.22,1c3686074548f93805135fcedb79475b
COVID-3482.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3482.png,COVID/images/COVID-3482.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3482.png,COVID/masks/COVID-3482.png,True,False,299,299,L,90.94,b21b01ef49643b4fdf58b8e363d654bc
COVID-3483.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3483.png,COVID/images/COVID-3483.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3483.png,COVID/masks/COVID-3483.png,True,False,299,299,L,144.99,e75d6ed4365ca7b34bcf475997ce325d
COVID-3484.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3484.png,COVID/images/COVID-3484.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3484.png,COVID/masks/COVID-3484.png,True,False,299,299,L,146.74,7fee3fb3a88b813a25881223e4bc9d76
COVID-3485.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3485.png,COVID/images/COVID-3485.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3485.png,COVID/masks/COVID-3485.png,True,False,299,299,L,71.03,963cb7672a28a6ad5ea7e30309dab585
COVID-3486.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3486.png,COVID/images/COVID-3486.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3486.png,COVID/masks/COVID-3486.png,True,False,299,299,L,73.68,95d140244b577ceab7b3a5e56571d66b
COVID-3487.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3487.png,COVID/images/COVID-3487.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3487.png,COVID/masks/COVID-3487.png,True,False,299,299,L,140.46,3c077ee5ed5b431d1704b7733b4145c6
COVID-3488.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3488.png,COVID/images/COVID-3488.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3488.png,COVID/masks/COVID-3488.png,True,False,299,299,L,137.4,147af16441cea2817b8a09c65c43915b
COVID-3489.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3489.png,COVID/images/COVID-3489.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3489.png,COVID/masks/COVID-3489.png,True,False,299,299,L,137.4,147af16441cea2817b8a09c65c43915b
COVID-349.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-349.png,COVID/images/COVID-349.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-349.png,COVID/masks/COVID-349.png,True,False,299,299,L,103.44,fdec468965978ad4b97c8a6d8ec19369
COVID-3490.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3490.png,COVID/images/COVID-3490.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3490.png,COVID/masks/COVID-3490.png,True,False,299,299,L,67.34,41dd1fd523fe0caa6586bb592ff096d3
COVID-3491.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3491.png,COVID/images/COVID-3491.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3491.png,COVID/masks/COVID-3491.png,True,False,299,299,L,145.55,a2e3aee13a029f1313033731c42e29e4
COVID-3492.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3492.png,COVID/images/COVID-3492.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3492.png,COVID/masks/COVID-3492.png,True,False,299,299,L,94.73,dd0fd7d2b2545e5fab7f12b7e6452663
COVID-3493.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3493.png,COVID/images/COVID-3493.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3493.png,COVID/masks/COVID-3493.png,True,False,299,299,L,172.35,a252b8b1da3f2c24913f1fbd8b3c3ca2
COVID-3494.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3494.png,COVID/images/COVID-3494.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3494.png,COVID/masks/COVID-3494.png,True,False,299,299,L,65.94,8290e98f92c4046f4c36429b9ed73642
COVID-3495.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3495.png,COVID/images/COVID-3495.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3495.png,COVID/masks/COVID-3495.png,True,False,299,299,L,127.77,6eb97ca92f90063c86547ec5d6b4f0c6
COVID-3496.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3496.png,COVID/images/COVID-3496.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3496.png,COVID/masks/COVID-3496.png,True,False,299,299,L,116.27,ccaa63f63a2df17428e3a13f37b4605a
COVID-3497.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3497.png,COVID/images/COVID-3497.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3497.png,COVID/masks/COVID-3497.png,True,False,299,299,L,105.83,68ee14fa1863e65b828854ed6ccb7b37
COVID-3498.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3498.png,COVID/images/COVID-3498.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3498.png,COVID/masks/COVID-3498.png,True,False,299,299,L,122.49,a19b05e9ca931e3d683588cfc43a827c
COVID-3499.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3499.png,COVID/images/COVID-3499.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3499.png,COVID/masks/COVID-3499.png,True,False,299,299,L,140.46,3698b763d2239443ed36443b0ddd0545
COVID-35.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-35.png,COVID/images/COVID-35.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-35.png,COVID/masks/COVID-35.png,True,False,299,299,L,137.18,c916d1a103bbe063916ddc5e9e346ca7
COVID-350.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-350.png,COVID/images/COVID-350.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-350.png,COVID/masks/COVID-350.png,True,False,299,299,L,129.22,c1a8d22fdf28f793cdee1a085380ff95
COVID-3500.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3500.png,COVID/images/COVID-3500.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3500.png,COVID/masks/COVID-3500.png,True,False,299,299,L,135.09,46803b86faf48bc1bed11e62f9c166ce
COVID-3501.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3501.png,COVID/images/COVID-3501.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3501.png,COVID/masks/COVID-3501.png,True,False,299,299,L,130.4,90751bd97472ca3fec57d93f3e6b761c
COVID-3502.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3502.png,COVID/images/COVID-3502.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3502.png,COVID/masks/COVID-3502.png,True,False,299,299,L,169.26,c77587cf3b7e4940a8a21447bb72af37
COVID-3503.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3503.png,COVID/images/COVID-3503.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3503.png,COVID/masks/COVID-3503.png,True,False,299,299,L,111.05,7e723433f91929377ec72348a43efec3
COVID-3504.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3504.png,COVID/images/COVID-3504.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3504.png,COVID/masks/COVID-3504.png,True,False,299,299,L,143.75,15d32fc6302351e9eb0fe62bc31b7eb5
COVID-3505.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3505.png,COVID/images/COVID-3505.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3505.png,COVID/masks/COVID-3505.png,True,False,299,299,L,151.03,e93c75d3217f5ab4636facd72ef7135b
COVID-3506.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3506.png,COVID/images/COVID-3506.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3506.png,COVID/masks/COVID-3506.png,True,False,299,299,L,107.02,8c7d1974e6097ef502fa19fe927405d7
COVID-3507.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3507.png,COVID/images/COVID-3507.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3507.png,COVID/masks/COVID-3507.png,True,False,299,299,L,64.99,93f78cd42b2c0559cd5e1da96f541ad0
COVID-3508.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3508.png,COVID/images/COVID-3508.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3508.png,COVID/masks/COVID-3508.png,True,False,299,299,L,144.17,63a6b29e70ad5d39ed1c132d897d644e
COVID-3509.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3509.png,COVID/images/COVID-3509.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3509.png,COVID/masks/COVID-3509.png,True,False,299,299,L,87.98,55ac132f18f0169168e9b7cb183983dc
COVID-351.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-351.png,COVID/images/COVID-351.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-351.png,COVID/masks/COVID-351.png,True,False,299,299,L,156.03,12f661349cda0059365b258d4295cae4
COVID-3510.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3510.png,COVID/images/COVID-3510.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3510.png,COVID/masks/COVID-3510.png,True,False,299,299,L,141.39,6741980dcebdc4857aa065f2ffd9b066
COVID-3511.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3511.png,COVID/images/COVID-3511.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3511.png,COVID/masks/COVID-3511.png,True,False,299,299,L,88.99,3e84651fcbd3b09564d2c223d2f4b8b1
COVID-3512.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3512.png,COVID/images/COVID-3512.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3512.png,COVID/masks/COVID-3512.png,True,False,299,299,L,132.88,45a4e37dc365db9b3dd10a6b96002a1e
COVID-3513.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3513.png,COVID/images/COVID-3513.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3513.png,COVID/masks/COVID-3513.png,True,False,299,299,L,134.44,80c47ca2ce1f9fa92d64de3ad8cf8c1c
COVID-3514.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3514.png,COVID/images/COVID-3514.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3514.png,COVID/masks/COVID-3514.png,True,False,299,299,L,160.3,3baec3e632c51abbdfdc291d39bf32de
COVID-3515.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3515.png,COVID/images/COVID-3515.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3515.png,COVID/masks/COVID-3515.png,True,False,299,299,L,134.0,d72a483022bea7307051bb6d2ced0a3b
COVID-3516.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3516.png,COVID/images/COVID-3516.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3516.png,COVID/masks/COVID-3516.png,True,False,299,299,L,134.0,d72a483022bea7307051bb6d2ced0a3b
COVID-3517.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3517.png,COVID/images/COVID-3517.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3517.png,COVID/masks/COVID-3517.png,True,False,299,299,L,150.48,8d2a6ae2067e7e4d08ecda82ab457d8d
COVID-3518.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3518.png,COVID/images/COVID-3518.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3518.png,COVID/masks/COVID-3518.png,True,False,299,299,L,117.81,49529aa58abe1bb76e1e49cba5362361
COVID-3519.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3519.png,COVID/images/COVID-3519.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3519.png,COVID/masks/COVID-3519.png,True,False,299,299,L,75.36,8b1c15908a58b6d0ee920279a89d099e
COVID-352.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-352.png,COVID/images/COVID-352.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-352.png,COVID/masks/COVID-352.png,True,False,299,299,L,134.3,73a5007caa7f7e894fdee1045d4f3e7d
COVID-3520.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3520.png,COVID/images/COVID-3520.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3520.png,COVID/masks/COVID-3520.png,True,False,299,299,L,131.8,a6d730acd561eb3ae24c831c5133c8da
COVID-3521.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3521.png,COVID/images/COVID-3521.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3521.png,COVID/masks/COVID-3521.png,True,False,299,299,L,94.54,92fa0bf50f14162bc8c5f96870710592
COVID-3522.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3522.png,COVID/images/COVID-3522.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3522.png,COVID/masks/COVID-3522.png,True,False,299,299,L,81.86,4daaf216f8f686a2b3974ae45c17a64f
COVID-3523.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3523.png,COVID/images/COVID-3523.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3523.png,COVID/masks/COVID-3523.png,True,False,299,299,L,114.85,892f87941ffa3aaf09c6c68b20943d15
COVID-3524.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3524.png,COVID/images/COVID-3524.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3524.png,COVID/masks/COVID-3524.png,True,False,299,299,L,65.74,98a30979264e5c98cfd1e0b2ecc77968
COVID-3525.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3525.png,COVID/images/COVID-3525.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3525.png,COVID/masks/COVID-3525.png,True,False,299,299,L,75.59,b4cb70d8044c0d9a8d4deb50375a3e8e
COVID-3526.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3526.png,COVID/images/COVID-3526.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3526.png,COVID/masks/COVID-3526.png,True,False,299,299,L,170.58,b096f623abbb4e63a0356d3201e09f8c
COVID-3527.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3527.png,COVID/images/COVID-3527.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3527.png,COVID/masks/COVID-3527.png,True,False,299,299,L,96.29,563bf42c9b2a40a265e9734ee0772ec2
COVID-3528.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3528.png,COVID/images/COVID-3528.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3528.png,COVID/masks/COVID-3528.png,True,False,299,299,L,117.04,23d17508ac3436182d9866d53fe387ca
COVID-3529.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3529.png,COVID/images/COVID-3529.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3529.png,COVID/masks/COVID-3529.png,True,False,299,299,L,143.1,7669b6c67f0702e93acd780e069c89fa
COVID-353.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-353.png,COVID/images/COVID-353.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-353.png,COVID/masks/COVID-353.png,True,False,299,299,L,153.14,e4034813e09d43346279da841fe05a4c
COVID-3530.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3530.png,COVID/images/COVID-3530.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3530.png,COVID/masks/COVID-3530.png,True,False,299,299,L,125.52,402ca77aa2f9414649844bc11bea8b07
COVID-3531.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3531.png,COVID/images/COVID-3531.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3531.png,COVID/masks/COVID-3531.png,True,False,299,299,L,134.83,ab08474191e073ae54d52268776ea002
COVID-3532.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3532.png,COVID/images/COVID-3532.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3532.png,COVID/masks/COVID-3532.png,True,False,299,299,L,108.61,54bbf733bcd8331cb8bf4043a51f8d12
COVID-3533.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3533.png,COVID/images/COVID-3533.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3533.png,COVID/masks/COVID-3533.png,True,False,299,299,L,115.31,c42baab30a2d9143d1440bd6190bf8e5
COVID-3534.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3534.png,COVID/images/COVID-3534.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3534.png,COVID/masks/COVID-3534.png,True,False,299,299,L,117.43,da9a8cf3ad924b3adcb43ff1ed940131
COVID-3535.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3535.png,COVID/images/COVID-3535.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3535.png,COVID/masks/COVID-3535.png,True,False,299,299,L,122.48,2cd886a283b162e37b202ecd551b7c18
COVID-3536.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3536.png,COVID/images/COVID-3536.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3536.png,COVID/masks/COVID-3536.png,True,False,299,299,L,165.01,e202900447d28e8ead3eeee42069d630
COVID-3537.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3537.png,COVID/images/COVID-3537.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3537.png,COVID/masks/COVID-3537.png,True,False,299,299,L,124.22,8fe07bad81b7a1e8c331496e75ebd3de
COVID-3538.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3538.png,COVID/images/COVID-3538.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3538.png,COVID/masks/COVID-3538.png,True,False,299,299,L,142.04,0ac0690903c4f8ea11ebd0ae991ffa19
COVID-3539.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3539.png,COVID/images/COVID-3539.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3539.png,COVID/masks/COVID-3539.png,True,False,299,299,L,66.16,9b1cca60a168454ab44a50e3ff8d7096
COVID-354.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-354.png,COVID/images/COVID-354.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-354.png,COVID/masks/COVID-354.png,True,False,299,299,L,155.53,eb7aa436f1b8ffe232a3ec816158d0f8
COVID-3540.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3540.png,COVID/images/COVID-3540.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3540.png,COVID/masks/COVID-3540.png,True,False,299,299,L,124.85,28c06f14e6d264d81f5f4e23a868592a
COVID-3541.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3541.png,COVID/images/COVID-3541.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3541.png,COVID/masks/COVID-3541.png,True,False,299,299,L,127.81,3dc756e5f87ed30fed728ac898f71fd4
COVID-3542.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3542.png,COVID/images/COVID-3542.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3542.png,COVID/masks/COVID-3542.png,True,False,299,299,L,78.95,feb144be4a599718406b39798591cae0
COVID-3543.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3543.png,COVID/images/COVID-3543.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3543.png,COVID/masks/COVID-3543.png,True,False,299,299,L,87.26,7508f5f82035c3e964c3827f82bada78
COVID-3544.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3544.png,COVID/images/COVID-3544.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3544.png,COVID/masks/COVID-3544.png,True,False,299,299,L,87.78,1e4fbf20396ef52d1514538d94aea88b
COVID-3545.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3545.png,COVID/images/COVID-3545.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3545.png,COVID/masks/COVID-3545.png,True,False,299,299,L,89.74,8aea855e4012028a55f0d7d3508e906d
COVID-3546.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3546.png,COVID/images/COVID-3546.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3546.png,COVID/masks/COVID-3546.png,True,False,299,299,L,139.59,cf10208e46859126e22a88164b6e5bcf
COVID-3547.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3547.png,COVID/images/COVID-3547.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3547.png,COVID/masks/COVID-3547.png,True,False,299,299,L,147.54,94a41317b4ce738e8526f54f9d281fec
COVID-3548.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3548.png,COVID/images/COVID-3548.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3548.png,COVID/masks/COVID-3548.png,True,False,299,299,L,147.54,4174f4449ec3f6c4c80a2aae15e69376
COVID-3549.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3549.png,COVID/images/COVID-3549.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3549.png,COVID/masks/COVID-3549.png,True,False,299,299,L,115.45,74060a698222132ecbbad031f651fd8f
COVID-355.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-355.png,COVID/images/COVID-355.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-355.png,COVID/masks/COVID-355.png,True,False,299,299,L,151.46,9ce4aab149975a6ba8ad274a874fcdb0
COVID-3550.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3550.png,COVID/images/COVID-3550.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3550.png,COVID/masks/COVID-3550.png,True,False,299,299,L,102.61,ad36f6aee4e10d901b8f54ec6360f896
COVID-3551.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3551.png,COVID/images/COVID-3551.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3551.png,COVID/masks/COVID-3551.png,True,False,299,299,L,143.8,072f7e724c2fe2e37d6a3721806be5b6
COVID-3552.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3552.png,COVID/images/COVID-3552.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3552.png,COVID/masks/COVID-3552.png,True,False,299,299,L,157.61,aacb5cc6141d07623c0c7b4ec1d0726c
COVID-3553.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3553.png,COVID/images/COVID-3553.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3553.png,COVID/masks/COVID-3553.png,True,False,299,299,L,142.87,be1ceaf8de72a34fe7b16fae4d0deaec
COVID-3554.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3554.png,COVID/images/COVID-3554.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3554.png,COVID/masks/COVID-3554.png,True,False,299,299,L,158.55,5d4f99d5e09ef619f22bddb27e0bc568
COVID-3555.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3555.png,COVID/images/COVID-3555.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3555.png,COVID/masks/COVID-3555.png,True,False,299,299,L,168.82,e052e5c771113b163bb5892625a775f2
COVID-3556.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3556.png,COVID/images/COVID-3556.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3556.png,COVID/masks/COVID-3556.png,True,False,299,299,L,136.0,ab69a600728cf4c72cbcfd630a7df04e
COVID-3557.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3557.png,COVID/images/COVID-3557.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3557.png,COVID/masks/COVID-3557.png,True,False,299,299,L,151.02,65565a672b1b751ab80cf7879b8d117f
COVID-3558.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3558.png,COVID/images/COVID-3558.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3558.png,COVID/masks/COVID-3558.png,True,False,299,299,L,137.63,c57c0add7da90b1df1578e23719331ca
COVID-3559.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3559.png,COVID/images/COVID-3559.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3559.png,COVID/masks/COVID-3559.png,True,False,299,299,L,171.19,efd4f69f69f75a35dd59e84d24aeccef
COVID-356.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-356.png,COVID/images/COVID-356.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-356.png,COVID/masks/COVID-356.png,True,False,299,299,L,147.83,46f744244206a110feafe8cfb1a94552
COVID-3560.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3560.png,COVID/images/COVID-3560.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3560.png,COVID/masks/COVID-3560.png,True,False,299,299,L,113.71,44d37a45900257e9f0644e7c424c789c
COVID-3561.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3561.png,COVID/images/COVID-3561.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3561.png,COVID/masks/COVID-3561.png,True,False,299,299,L,146.42,f39da6bc4c7550c7e396fed0d5658b93
COVID-3562.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3562.png,COVID/images/COVID-3562.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3562.png,COVID/masks/COVID-3562.png,True,False,299,299,L,99.99,523f625ead35ae81d59c2edf1bc60975
COVID-3563.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3563.png,COVID/images/COVID-3563.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3563.png,COVID/masks/COVID-3563.png,True,False,299,299,L,129.67,c1faa2d17e32ecb76297b979eff19c50
COVID-3564.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3564.png,COVID/images/COVID-3564.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3564.png,COVID/masks/COVID-3564.png,True,False,299,299,L,101.25,41bcba060a7a9fa19032ce61bc324877
COVID-3565.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3565.png,COVID/images/COVID-3565.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3565.png,COVID/masks/COVID-3565.png,True,False,299,299,L,99.61,5cefd8f77ed2d0e4450a8232f9598615
COVID-3566.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3566.png,COVID/images/COVID-3566.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3566.png,COVID/masks/COVID-3566.png,True,False,299,299,L,102.08,5604314c6a102129565a7c0660175e6e
COVID-3567.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3567.png,COVID/images/COVID-3567.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3567.png,COVID/masks/COVID-3567.png,True,False,299,299,L,78.05,9db2842560382c691550d2e049e76ff7
COVID-3568.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3568.png,COVID/images/COVID-3568.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3568.png,COVID/masks/COVID-3568.png,True,False,299,299,L,165.04,9bc503ac3b45ff114aa8976f7ab5af7c
COVID-3569.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3569.png,COVID/images/COVID-3569.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3569.png,COVID/masks/COVID-3569.png,True,False,299,299,L,75.18,5787b7d9622d6e245e8a4f654c812beb
COVID-357.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-357.png,COVID/images/COVID-357.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-357.png,COVID/masks/COVID-357.png,True,False,299,299,L,93.85,1811f31b582f8fde06dc4aafccbca421
COVID-3570.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3570.png,COVID/images/COVID-3570.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3570.png,COVID/masks/COVID-3570.png,True,False,299,299,L,155.94,7e941029ef9dc3a5fc3e18c14bd4413e
COVID-3571.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3571.png,COVID/images/COVID-3571.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3571.png,COVID/masks/COVID-3571.png,True,False,299,299,L,163.8,86399a95f790bb94ae923be00f3cf32c
COVID-3572.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3572.png,COVID/images/COVID-3572.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3572.png,COVID/masks/COVID-3572.png,True,False,299,299,L,80.36,c61ef24e38b0863def44ed6f98aeafbe
COVID-3573.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3573.png,COVID/images/COVID-3573.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3573.png,COVID/masks/COVID-3573.png,True,False,299,299,L,150.41,b2a4a8615eb6bbc763cb5a58c50f4386
COVID-3574.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3574.png,COVID/images/COVID-3574.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3574.png,COVID/masks/COVID-3574.png,True,False,299,299,L,150.41,b2a4a8615eb6bbc763cb5a58c50f4386
COVID-3575.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3575.png,COVID/images/COVID-3575.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3575.png,COVID/masks/COVID-3575.png,True,False,299,299,L,101.89,8df277dfa6478c24ba298487e226e77a
COVID-3576.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3576.png,COVID/images/COVID-3576.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3576.png,COVID/masks/COVID-3576.png,True,False,299,299,L,114.81,e50b368197525c1f690e5a2dbc8f90df
COVID-3577.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3577.png,COVID/images/COVID-3577.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3577.png,COVID/masks/COVID-3577.png,True,False,299,299,L,111.41,1a7e092e49893e6e7767e9a9c553530a
COVID-3578.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3578.png,COVID/images/COVID-3578.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3578.png,COVID/masks/COVID-3578.png,True,False,299,299,L,154.16,0989fa845e9ba40c6a289619c36637a6
COVID-3579.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3579.png,COVID/images/COVID-3579.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3579.png,COVID/masks/COVID-3579.png,True,False,299,299,L,137.32,7f52331416839a72ecd6c8fda8337ae3
COVID-358.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-358.png,COVID/images/COVID-358.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-358.png,COVID/masks/COVID-358.png,True,False,299,299,L,156.85,e1685bbfee832287d849c75fbe62cef4
COVID-3580.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3580.png,COVID/images/COVID-3580.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3580.png,COVID/masks/COVID-3580.png,True,False,299,299,L,114.04,b89df5f2475ec4fd363ce8788d895844
COVID-3581.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3581.png,COVID/images/COVID-3581.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3581.png,COVID/masks/COVID-3581.png,True,False,299,299,L,117.45,08e5868a81cd5e8bd79676e72f80f240
COVID-3582.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3582.png,COVID/images/COVID-3582.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3582.png,COVID/masks/COVID-3582.png,True,False,299,299,L,110.29,a946b88001a62cb878cd05df2337342b
COVID-3583.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3583.png,COVID/images/COVID-3583.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3583.png,COVID/masks/COVID-3583.png,True,False,299,299,L,94.1,01891861c13577dab8cdd10652f6506f
COVID-3584.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3584.png,COVID/images/COVID-3584.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3584.png,COVID/masks/COVID-3584.png,True,False,299,299,L,152.44,20316db6c04f0e9ee4543cf38b27b4b6
COVID-3585.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3585.png,COVID/images/COVID-3585.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3585.png,COVID/masks/COVID-3585.png,True,False,299,299,L,108.51,d455017470a2efcc9c9b9ab6c736f425
COVID-3586.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3586.png,COVID/images/COVID-3586.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3586.png,COVID/masks/COVID-3586.png,True,False,299,299,L,86.89,b7e9673ec51e023ffb0b9604c5816d24
COVID-3587.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3587.png,COVID/images/COVID-3587.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3587.png,COVID/masks/COVID-3587.png,True,False,299,299,L,153.52,54fe0093dfffce1326b6f55694362d17
COVID-3588.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3588.png,COVID/images/COVID-3588.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3588.png,COVID/masks/COVID-3588.png,True,False,299,299,L,87.06,3cbb67bb072a12c0566911f1838ec030
COVID-3589.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3589.png,COVID/images/COVID-3589.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3589.png,COVID/masks/COVID-3589.png,True,False,299,299,L,155.34,36ae7ab7eb41ee0509914730772989f4
COVID-359.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-359.png,COVID/images/COVID-359.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-359.png,COVID/masks/COVID-359.png,True,False,299,299,L,165.41,8779347c1dc7d3835a55667b3b07f358
COVID-3590.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3590.png,COVID/images/COVID-3590.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3590.png,COVID/masks/COVID-3590.png,True,False,299,299,L,130.68,9930b01a3eee1cb5cef2105eb56fd3e5
COVID-3591.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3591.png,COVID/images/COVID-3591.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3591.png,COVID/masks/COVID-3591.png,True,False,299,299,L,130.68,9930b01a3eee1cb5cef2105eb56fd3e5
COVID-3592.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3592.png,COVID/images/COVID-3592.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3592.png,COVID/masks/COVID-3592.png,True,False,299,299,L,137.8,1538401f9c69fdb05a3a69d9c54eb649
COVID-3593.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3593.png,COVID/images/COVID-3593.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3593.png,COVID/masks/COVID-3593.png,True,False,299,299,L,122.35,8fe5726ca18ec04ab1529d4abf8126a5
COVID-3594.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3594.png,COVID/images/COVID-3594.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3594.png,COVID/masks/COVID-3594.png,True,False,299,299,L,162.09,77f6c3557b335207684fc7dbd38830f6
COVID-3595.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3595.png,COVID/images/COVID-3595.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3595.png,COVID/masks/COVID-3595.png,True,False,299,299,L,126.61,196aabab039e9ba2c7fe162346c568a5
COVID-3596.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3596.png,COVID/images/COVID-3596.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3596.png,COVID/masks/COVID-3596.png,True,False,299,299,L,161.38,76551ebdbc5aab9b52d24c91d0663e4e
COVID-3597.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3597.png,COVID/images/COVID-3597.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3597.png,COVID/masks/COVID-3597.png,True,False,299,299,L,100.61,52b0a428eb36388f3334b587ad6ca081
COVID-3598.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3598.png,COVID/images/COVID-3598.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3598.png,COVID/masks/COVID-3598.png,True,False,299,299,L,75.13,ff111211d450f12698dffb5cb7f9067f
COVID-3599.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3599.png,COVID/images/COVID-3599.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3599.png,COVID/masks/COVID-3599.png,True,False,299,299,L,75.13,ff111211d450f12698dffb5cb7f9067f
COVID-36.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-36.png,COVID/images/COVID-36.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-36.png,COVID/masks/COVID-36.png,True,False,299,299,L,153.96,e49f77845437e61080de219b859f827f
COVID-360.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-360.png,COVID/images/COVID-360.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-360.png,COVID/masks/COVID-360.png,True,False,299,299,L,101.73,893c70c4364a1956a52eb4999c27c037
COVID-3600.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3600.png,COVID/images/COVID-3600.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3600.png,COVID/masks/COVID-3600.png,True,False,299,299,L,81.81,67129dbaa6dc48b3dd3fe84e67b2f4c3
COVID-3601.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3601.png,COVID/images/COVID-3601.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3601.png,COVID/masks/COVID-3601.png,True,False,299,299,L,99.25,bb08844f858bd907b4a2bff3babc3829
COVID-3602.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3602.png,COVID/images/COVID-3602.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3602.png,COVID/masks/COVID-3602.png,True,False,299,299,L,133.76,f695e0a6894cc2b5d1db40d873c6d5d5
COVID-3603.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3603.png,COVID/images/COVID-3603.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3603.png,COVID/masks/COVID-3603.png,True,False,299,299,L,117.1,aefa1642343630c23c31148495f296ce
COVID-3604.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3604.png,COVID/images/COVID-3604.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3604.png,COVID/masks/COVID-3604.png,True,False,299,299,L,106.37,94b786b2a6d5d3b9717822be7005f4eb
COVID-3605.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3605.png,COVID/images/COVID-3605.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3605.png,COVID/masks/COVID-3605.png,True,False,299,299,L,142.73,4fa785ecfc6963fdc1e35b96acb62b2a
COVID-3606.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3606.png,COVID/images/COVID-3606.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3606.png,COVID/masks/COVID-3606.png,True,False,299,299,L,102.31,c04e13cc6e1c124541ecf49274667804
COVID-3607.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3607.png,COVID/images/COVID-3607.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3607.png,COVID/masks/COVID-3607.png,True,False,299,299,L,121.55,7051b7c9e3ea278c242544368189f936
COVID-3608.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3608.png,COVID/images/COVID-3608.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3608.png,COVID/masks/COVID-3608.png,True,False,299,299,L,110.47,f505ef7e8acc673d5d7b341409b86faf
COVID-3609.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3609.png,COVID/images/COVID-3609.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3609.png,COVID/masks/COVID-3609.png,True,False,299,299,L,115.23,93c72da9a1405981d4526e0d872cc1a1
COVID-361.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-361.png,COVID/images/COVID-361.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-361.png,COVID/masks/COVID-361.png,True,False,299,299,L,151.81,fd35977987430f31822eba374b4ff60d
COVID-3610.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3610.png,COVID/images/COVID-3610.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3610.png,COVID/masks/COVID-3610.png,True,False,299,299,L,136.42,910d7b29bf66f28af4ce02ac9f5f1d9b
COVID-3611.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3611.png,COVID/images/COVID-3611.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3611.png,COVID/masks/COVID-3611.png,True,False,299,299,L,149.25,b1989571b32d22c90f11275356503731
COVID-3612.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3612.png,COVID/images/COVID-3612.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3612.png,COVID/masks/COVID-3612.png,True,False,299,299,L,113.15,2a7f77bd7ea430c4618a3c74494ec61f
COVID-3613.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3613.png,COVID/images/COVID-3613.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3613.png,COVID/masks/COVID-3613.png,True,False,299,299,L,163.82,c34c4f8c85b300956d17e346a85ac4b0
COVID-3614.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3614.png,COVID/images/COVID-3614.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3614.png,COVID/masks/COVID-3614.png,True,False,299,299,L,112.79,fc092f11865d4a501f9697b599415be1
COVID-3615.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3615.png,COVID/images/COVID-3615.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3615.png,COVID/masks/COVID-3615.png,True,False,299,299,L,25.51,d8afafea7269cd10ddbbed7e255f3124
COVID-3616.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-3616.png,COVID/images/COVID-3616.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-3616.png,COVID/masks/COVID-3616.png,True,False,299,299,L,102.87,144acd7136e367938a3601deeee0b68d
COVID-362.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-362.png,COVID/images/COVID-362.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-362.png,COVID/masks/COVID-362.png,True,False,299,299,L,156.46,d263c1b2b5ce155fa0e2faf64b15699d
COVID-363.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-363.png,COVID/images/COVID-363.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-363.png,COVID/masks/COVID-363.png,True,False,299,299,L,110.18,090283788af8d04464a39d4a35ac2707
COVID-364.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-364.png,COVID/images/COVID-364.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-364.png,COVID/masks/COVID-364.png,True,False,299,299,L,171.49,dc0fcd722a0721296fdc93398f95385b
COVID-365.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-365.png,COVID/images/COVID-365.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-365.png,COVID/masks/COVID-365.png,True,False,299,299,L,170.68,b5ca54ad909bee332064e10300e40721
COVID-366.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-366.png,COVID/images/COVID-366.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-366.png,COVID/masks/COVID-366.png,True,False,299,299,L,105.7,c3628bdd72ffb3263976781565484f76
COVID-367.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-367.png,COVID/images/COVID-367.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-367.png,COVID/masks/COVID-367.png,True,False,299,299,L,163.47,c5848dfccf9f1e38f7bfe85e92e4757c
COVID-368.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-368.png,COVID/images/COVID-368.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-368.png,COVID/masks/COVID-368.png,True,False,299,299,L,170.52,ca7500222aec162817ede603844014f1
COVID-369.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-369.png,COVID/images/COVID-369.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-369.png,COVID/masks/COVID-369.png,True,False,299,299,L,143.95,445581cdd924cf1e38c31a5c1a992af2
COVID-37.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-37.png,COVID/images/COVID-37.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-37.png,COVID/masks/COVID-37.png,True,False,299,299,L,90.29,6204574668a283eff4588d2d114bd48e
COVID-370.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-370.png,COVID/images/COVID-370.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-370.png,COVID/masks/COVID-370.png,True,False,299,299,L,146.04,bfe313f793769ddb7927349d8edbb240
COVID-371.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-371.png,COVID/images/COVID-371.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-371.png,COVID/masks/COVID-371.png,True,False,299,299,L,172.2,71e7b9b2279c6dc718db6809f1f90410
COVID-372.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-372.png,COVID/images/COVID-372.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-372.png,COVID/masks/COVID-372.png,True,False,299,299,L,128.52,e908ace55b3ec9ff29351e16f8789f75
COVID-373.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-373.png,COVID/images/COVID-373.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-373.png,COVID/masks/COVID-373.png,True,False,299,299,L,145.05,21740822774283788ccea87b4293725a
COVID-374.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-374.png,COVID/images/COVID-374.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-374.png,COVID/masks/COVID-374.png,True,False,299,299,L,166.7,1c3f1919a34f1172d068104b85911748
COVID-375.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-375.png,COVID/images/COVID-375.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-375.png,COVID/masks/COVID-375.png,True,False,299,299,L,155.41,d0210c268654eda940b62ca445bac6e0
COVID-376.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-376.png,COVID/images/COVID-376.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-376.png,COVID/masks/COVID-376.png,True,False,299,299,L,135.48,523362f6ce2ea8c1e716b051a1f431bf
COVID-377.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-377.png,COVID/images/COVID-377.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-377.png,COVID/masks/COVID-377.png,True,False,299,299,L,155.92,d6742f8973f78964328fa29d213734b8
COVID-378.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-378.png,COVID/images/COVID-378.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-378.png,COVID/masks/COVID-378.png,True,False,299,299,L,139.57,4b7e22124f0188b204cd21db11f70ede
COVID-379.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-379.png,COVID/images/COVID-379.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-379.png,COVID/masks/COVID-379.png,True,False,299,299,L,125.82,80e233ce16491e4ce1f8c1802a2aa80e
COVID-38.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-38.png,COVID/images/COVID-38.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-38.png,COVID/masks/COVID-38.png,True,False,299,299,L,135.67,5a5652289c0c0eaae5b85bf12b857f62
COVID-380.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-380.png,COVID/images/COVID-380.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-380.png,COVID/masks/COVID-380.png,True,False,299,299,L,148.05,53c5d37f54f685a82bf971d111f98657
COVID-381.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-381.png,COVID/images/COVID-381.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-381.png,COVID/masks/COVID-381.png,True,False,299,299,L,111.79,af277a7689d13ec583cc03eb0e5c78dc
COVID-382.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-382.png,COVID/images/COVID-382.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-382.png,COVID/masks/COVID-382.png,True,False,299,299,L,150.55,b85fbec54d9d3f328165b70175dc1653
COVID-383.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-383.png,COVID/images/COVID-383.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-383.png,COVID/masks/COVID-383.png,True,False,299,299,L,111.4,5e4d2eb8d79e791ac9b5cefa1836ee4f
COVID-384.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-384.png,COVID/images/COVID-384.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-384.png,COVID/masks/COVID-384.png,True,False,299,299,L,146.03,9b1b5ad908dc363422bba4af16b735d6
COVID-385.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-385.png,COVID/images/COVID-385.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-385.png,COVID/masks/COVID-385.png,True,False,299,299,L,102.89,d91e806e7b8bf1047046333a44612ab9
COVID-386.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-386.png,COVID/images/COVID-386.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-386.png,COVID/masks/COVID-386.png,True,False,299,299,L,134.46,1276001c5bcc6941eef425af448e4366
COVID-387.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-387.png,COVID/images/COVID-387.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-387.png,COVID/masks/COVID-387.png,True,False,299,299,L,137.06,45abb7f0b2a67d6e397716e4928fd52a
COVID-388.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-388.png,COVID/images/COVID-388.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-388.png,COVID/masks/COVID-388.png,True,False,299,299,L,152.49,a76b1531f401317851bc835ed9f03a22
COVID-389.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-389.png,COVID/images/COVID-389.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-389.png,COVID/masks/COVID-389.png,True,False,299,299,L,129.39,5b06c51d73d36aff17d1ac6178fce701
COVID-39.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-39.png,COVID/images/COVID-39.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-39.png,COVID/masks/COVID-39.png,True,False,299,299,L,150.99,60eabc85bef8196c504852103cdc79bc
COVID-390.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-390.png,COVID/images/COVID-390.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-390.png,COVID/masks/COVID-390.png,True,False,299,299,L,132.04,3970e086e4f3b5dd922570f5380966fd
COVID-391.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-391.png,COVID/images/COVID-391.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-391.png,COVID/masks/COVID-391.png,True,False,299,299,L,102.7,e73b556e54d6a67db9f89f19e7fb0735
COVID-392.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-392.png,COVID/images/COVID-392.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-392.png,COVID/masks/COVID-392.png,True,False,299,299,L,147.28,29d08ff45b964f460cc8f6d4b595b1bf
COVID-393.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-393.png,COVID/images/COVID-393.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-393.png,COVID/masks/COVID-393.png,True,False,299,299,L,142.04,adfff967aa1c1d7146d0d37465d31153
COVID-394.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-394.png,COVID/images/COVID-394.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-394.png,COVID/masks/COVID-394.png,True,False,299,299,L,135.1,609a7ea8d0f3943e646235b948815d22
COVID-395.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-395.png,COVID/images/COVID-395.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-395.png,COVID/masks/COVID-395.png,True,False,299,299,L,121.18,8a58314756e96a883c4880d41bf20854
COVID-396.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-396.png,COVID/images/COVID-396.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-396.png,COVID/masks/COVID-396.png,True,False,299,299,L,156.31,9ac17e169537609c687fa3309d0c4811
COVID-397.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-397.png,COVID/images/COVID-397.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-397.png,COVID/masks/COVID-397.png,True,False,299,299,L,121.97,b7afc6036ac24708dd5509763d9e95f9
COVID-398.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-398.png,COVID/images/COVID-398.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-398.png,COVID/masks/COVID-398.png,True,False,299,299,L,146.4,b5f0403d1e12fddc49b0768ef4aff315
COVID-399.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-399.png,COVID/images/COVID-399.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-399.png,COVID/masks/COVID-399.png,True,False,299,299,L,193.77,a26f04d07e54b2af6e5f41f7f83c1a7f
COVID-4.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-4.png,COVID/images/COVID-4.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-4.png,COVID/masks/COVID-4.png,True,False,299,299,L,116.71,023dd886818373be8dfbd50cb9579d05
COVID-40.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-40.png,COVID/images/COVID-40.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-40.png,COVID/masks/COVID-40.png,True,False,299,299,L,170.37,8922cf8ebf45e8eefb8e4620bd1fa8f6
COVID-400.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-400.png,COVID/images/COVID-400.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-400.png,COVID/masks/COVID-400.png,True,False,299,299,L,117.31,1100cce69c7039c0af407f92620f799f
COVID-401.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-401.png,COVID/images/COVID-401.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-401.png,COVID/masks/COVID-401.png,True,False,299,299,L,121.76,7276d821f2436756efebf6713a5b4f20
COVID-402.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-402.png,COVID/images/COVID-402.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-402.png,COVID/masks/COVID-402.png,True,False,299,299,L,183.01,f2929e4d0d264193cb48a6529d582fc6
COVID-403.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-403.png,COVID/images/COVID-403.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-403.png,COVID/masks/COVID-403.png,True,False,299,299,L,120.35,2d555605c6977a09c9c149b90d189e40
COVID-404.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-404.png,COVID/images/COVID-404.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-404.png,COVID/masks/COVID-404.png,True,False,299,299,L,166.8,f40472038a4c7ce7345d65802e0e4d07
COVID-405.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-405.png,COVID/images/COVID-405.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-405.png,COVID/masks/COVID-405.png,True,False,299,299,L,128.02,3249fb4a70850bf58572053bea9cd43a
COVID-406.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-406.png,COVID/images/COVID-406.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-406.png,COVID/masks/COVID-406.png,True,False,299,299,L,158.33,4ca5a2d735e4e64d9a1545aba1b8bd70
COVID-407.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-407.png,COVID/images/COVID-407.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-407.png,COVID/masks/COVID-407.png,True,False,299,299,L,153.79,cf6f4cd9c84e02426afa71c2837159b4
COVID-408.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-408.png,COVID/images/COVID-408.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-408.png,COVID/masks/COVID-408.png,True,False,299,299,L,139.39,fe3a6bd142fd57ce462e80e499fdcac3
COVID-409.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-409.png,COVID/images/COVID-409.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-409.png,COVID/masks/COVID-409.png,True,False,299,299,L,128.34,0401a090be86900ac157f33016da04cf
COVID-41.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-41.png,COVID/images/COVID-41.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-41.png,COVID/masks/COVID-41.png,True,False,299,299,L,137.28,8479662f7e550eb62e30a973619489ed
COVID-410.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-410.png,COVID/images/COVID-410.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-410.png,COVID/masks/COVID-410.png,True,False,299,299,L,148.65,968d6b18cee682c26de61856d260a018
COVID-411.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-411.png,COVID/images/COVID-411.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-411.png,COVID/masks/COVID-411.png,True,False,299,299,L,151.23,4d62be47d8b8cd8caa1d502412d994ae
COVID-412.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-412.png,COVID/images/COVID-412.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-412.png,COVID/masks/COVID-412.png,True,False,299,299,L,158.54,9027678ea233f337e2a9d557aa0fa817
COVID-413.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-413.png,COVID/images/COVID-413.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-413.png,COVID/masks/COVID-413.png,True,False,299,299,L,160.18,31fa2c7d45f2fd1e82867fba6da62251
COVID-414.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-414.png,COVID/images/COVID-414.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-414.png,COVID/masks/COVID-414.png,True,False,299,299,L,146.35,7bbbeddc7f424bd9dda05d3dc6ebf420
COVID-415.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-415.png,COVID/images/COVID-415.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-415.png,COVID/masks/COVID-415.png,True,False,299,299,L,154.75,a78af2d880ffb8520e41b210b7742c94
COVID-416.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-416.png,COVID/images/COVID-416.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-416.png,COVID/masks/COVID-416.png,True,False,299,299,L,152.12,dbaf6813d5288be9ae44942227de650e
COVID-417.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-417.png,COVID/images/COVID-417.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-417.png,COVID/masks/COVID-417.png,True,False,299,299,L,119.68,3f88427f00d32b999fbd628f3915a9ae
COVID-418.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-418.png,COVID/images/COVID-418.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-418.png,COVID/masks/COVID-418.png,True,False,299,299,L,153.14,c1bb869241df686d89b7960787c60302
COVID-419.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-419.png,COVID/images/COVID-419.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-419.png,COVID/masks/COVID-419.png,True,False,299,299,L,152.77,c071529d04a1608a705ddc741322ef2e
COVID-42.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-42.png,COVID/images/COVID-42.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-42.png,COVID/masks/COVID-42.png,True,False,299,299,L,130.96,9388d0e7d2cf481d96ecf5677baaf89d
COVID-420.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-420.png,COVID/images/COVID-420.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-420.png,COVID/masks/COVID-420.png,True,False,299,299,L,143.42,741f9428a530bd7acae6576655f2d311
COVID-421.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-421.png,COVID/images/COVID-421.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-421.png,COVID/masks/COVID-421.png,True,False,299,299,L,142.08,be0481490f497f67d5e0074dcfa354a2
COVID-422.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-422.png,COVID/images/COVID-422.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-422.png,COVID/masks/COVID-422.png,True,False,299,299,L,158.36,4d3d08c5b8177f2ec29a8f85a3c2b6e2
COVID-423.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-423.png,COVID/images/COVID-423.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-423.png,COVID/masks/COVID-423.png,True,False,299,299,L,118.39,55e8a83c51faa1b91ec36eed65e472da
COVID-424.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-424.png,COVID/images/COVID-424.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-424.png,COVID/masks/COVID-424.png,True,False,299,299,L,154.23,72d3685666d2a72b9eaa5b83075844ee
COVID-425.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-425.png,COVID/images/COVID-425.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-425.png,COVID/masks/COVID-425.png,True,False,299,299,L,135.98,0f7612824263c060645334672ceb74a2
COVID-426.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-426.png,COVID/images/COVID-426.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-426.png,COVID/masks/COVID-426.png,True,False,299,299,L,164.17,fe241f52115e338a93222698a0be1a9e
COVID-427.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-427.png,COVID/images/COVID-427.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-427.png,COVID/masks/COVID-427.png,True,False,299,299,L,151.1,0628e2738ac46de00bad789bf8034478
COVID-428.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-428.png,COVID/images/COVID-428.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-428.png,COVID/masks/COVID-428.png,True,False,299,299,L,141.52,a48aafa0848167c39b99f17d242ea7cc
COVID-429.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-429.png,COVID/images/COVID-429.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-429.png,COVID/masks/COVID-429.png,True,False,299,299,L,144.68,876865010b3f4bf9020dac888b968168
COVID-43.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-43.png,COVID/images/COVID-43.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-43.png,COVID/masks/COVID-43.png,True,False,299,299,L,123.37,f0b2440cdbf896947499fe56af311c04
COVID-430.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-430.png,COVID/images/COVID-430.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-430.png,COVID/masks/COVID-430.png,True,False,299,299,L,124.84,e4eebbdf83579585bbf4facc6183a6bb
COVID-431.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-431.png,COVID/images/COVID-431.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-431.png,COVID/masks/COVID-431.png,True,False,299,299,L,136.08,ae37d72fbb2ca21a8dfe35c9c925d5f7
COVID-432.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-432.png,COVID/images/COVID-432.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-432.png,COVID/masks/COVID-432.png,True,False,299,299,L,149.55,516f5c5993806f950a4a766ea4f1b5d4
COVID-433.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-433.png,COVID/images/COVID-433.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-433.png,COVID/masks/COVID-433.png,True,False,299,299,L,154.52,c8afce66138da0d8ba050f132bf18569
COVID-434.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-434.png,COVID/images/COVID-434.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-434.png,COVID/masks/COVID-434.png,True,False,299,299,L,144.29,74196c683ffb930db801194788dc1867
COVID-435.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-435.png,COVID/images/COVID-435.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-435.png,COVID/masks/COVID-435.png,True,False,299,299,L,118.64,79987ba3ba44cf1e1bc6c18ef5d381c2
COVID-436.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-436.png,COVID/images/COVID-436.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-436.png,COVID/masks/COVID-436.png,True,False,299,299,L,129.17,347f8472e19b80fe4180c3fc53f858a5
COVID-437.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-437.png,COVID/images/COVID-437.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-437.png,COVID/masks/COVID-437.png,True,False,299,299,L,127.7,5ff193ddf92b84a4eccf9132a1b76e5b
COVID-438.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-438.png,COVID/images/COVID-438.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-438.png,COVID/masks/COVID-438.png,True,False,299,299,L,160.23,5db45098051ce9acb19bc5a7bdf3627b
COVID-439.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-439.png,COVID/images/COVID-439.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-439.png,COVID/masks/COVID-439.png,True,False,299,299,L,164.05,8d26eba178d59a33ca00b924a880b102
COVID-44.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-44.png,COVID/images/COVID-44.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-44.png,COVID/masks/COVID-44.png,True,False,299,299,L,131.09,b37fdd167799634bc8338dbf16524dc4
COVID-440.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-440.png,COVID/images/COVID-440.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-440.png,COVID/masks/COVID-440.png,True,False,299,299,L,127.58,24fa0195da4757a37cc4b832fee02ce6
COVID-441.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-441.png,COVID/images/COVID-441.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-441.png,COVID/masks/COVID-441.png,True,False,299,299,L,139.83,24c066bae3528ef6eddc7bf655008659
COVID-442.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-442.png,COVID/images/COVID-442.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-442.png,COVID/masks/COVID-442.png,True,False,299,299,L,139.88,1d07d91524ce450805aa869329bcae29
COVID-443.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-443.png,COVID/images/COVID-443.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-443.png,COVID/masks/COVID-443.png,True,False,299,299,L,128.65,718cd9fabac0e2d73528f22f1ef33f97
COVID-444.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-444.png,COVID/images/COVID-444.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-444.png,COVID/masks/COVID-444.png,True,False,299,299,L,126.99,7c598b809d25a46d30475353616e59e9
COVID-445.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-445.png,COVID/images/COVID-445.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-445.png,COVID/masks/COVID-445.png,True,False,299,299,L,154.1,d1fe740c2354a45916569ab1d0f530b2
COVID-446.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-446.png,COVID/images/COVID-446.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-446.png,COVID/masks/COVID-446.png,True,False,299,299,L,85.16,ae7e9b121c6ecb956122eb3bf60dfdac
COVID-447.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-447.png,COVID/images/COVID-447.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-447.png,COVID/masks/COVID-447.png,True,False,299,299,L,159.13,b75065717183168c7b97026d0215631f
COVID-448.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-448.png,COVID/images/COVID-448.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-448.png,COVID/masks/COVID-448.png,True,False,299,299,L,150.38,163509e7d7845d636aedb1231671c894
COVID-449.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-449.png,COVID/images/COVID-449.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-449.png,COVID/masks/COVID-449.png,True,False,299,299,L,99.76,67bb59a4441d775c74bb2603586d9b78
COVID-45.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-45.png,COVID/images/COVID-45.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-45.png,COVID/masks/COVID-45.png,True,False,299,299,L,129.29,26fdc2e494e7113b8227538e7256f47d
COVID-450.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-450.png,COVID/images/COVID-450.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-450.png,COVID/masks/COVID-450.png,True,False,299,299,L,137.87,8967dd433804a2218d60ddc841eb7494
COVID-451.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-451.png,COVID/images/COVID-451.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-451.png,COVID/masks/COVID-451.png,True,False,299,299,L,95.33,2a2d93810ce6b98e859c0c548657f5ad
COVID-452.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-452.png,COVID/images/COVID-452.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-452.png,COVID/masks/COVID-452.png,True,False,299,299,L,139.13,f6a1a118ee4d8cec6f3f9e6136249592
COVID-453.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-453.png,COVID/images/COVID-453.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-453.png,COVID/masks/COVID-453.png,True,False,299,299,L,136.52,7284b5ded4258402a21c2ad4544444c8
COVID-454.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-454.png,COVID/images/COVID-454.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-454.png,COVID/masks/COVID-454.png,True,False,299,299,L,153.74,7294ba5f5080a01d751d0a2c024667c0
COVID-455.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-455.png,COVID/images/COVID-455.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-455.png,COVID/masks/COVID-455.png,True,False,299,299,L,141.73,8200c8cc64c506309991c470705e08f8
COVID-456.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-456.png,COVID/images/COVID-456.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-456.png,COVID/masks/COVID-456.png,True,False,299,299,L,143.85,4ed5b2bac107c071b4ff61a1bd479ec9
COVID-457.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-457.png,COVID/images/COVID-457.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-457.png,COVID/masks/COVID-457.png,True,False,299,299,L,142.26,323e816aa658033b1f59e8e96106a8d1
COVID-458.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-458.png,COVID/images/COVID-458.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-458.png,COVID/masks/COVID-458.png,True,False,299,299,L,150.94,a051475ec87c40a6a3327a27511bf8e3
COVID-459.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-459.png,COVID/images/COVID-459.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-459.png,COVID/masks/COVID-459.png,True,False,299,299,L,126.31,beb6f7a7bd3043baa08232e648c7c50d
COVID-46.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-46.png,COVID/images/COVID-46.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-46.png,COVID/masks/COVID-46.png,True,False,299,299,L,153.23,c4c02520dccae5f8b4253bf5a6d26052
COVID-460.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-460.png,COVID/images/COVID-460.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-460.png,COVID/masks/COVID-460.png,True,False,299,299,L,153.0,8876ad59b571b6a2f188cb27717854c1
COVID-461.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-461.png,COVID/images/COVID-461.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-461.png,COVID/masks/COVID-461.png,True,False,299,299,L,109.79,3880646355c8ec316d252b7739f4b8db
COVID-462.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-462.png,COVID/images/COVID-462.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-462.png,COVID/masks/COVID-462.png,True,False,299,299,L,139.58,66b5c65e52c092777854ecc4929a0d5c
COVID-463.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-463.png,COVID/images/COVID-463.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-463.png,COVID/masks/COVID-463.png,True,False,299,299,L,137.5,a13b485f57297b8afc63ee5535f8878e
COVID-464.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-464.png,COVID/images/COVID-464.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-464.png,COVID/masks/COVID-464.png,True,False,299,299,L,105.27,d8fed92e53a21de09bca245095e3a84c
COVID-465.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-465.png,COVID/images/COVID-465.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-465.png,COVID/masks/COVID-465.png,True,False,299,299,L,170.32,56fb22ee2a0087c95330ad059862948c
COVID-466.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-466.png,COVID/images/COVID-466.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-466.png,COVID/masks/COVID-466.png,True,False,299,299,L,137.23,a8a93f419fa2a0c254d67e884cda92d7
COVID-467.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-467.png,COVID/images/COVID-467.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-467.png,COVID/masks/COVID-467.png,True,False,299,299,L,140.86,755c13d467a21cf7a987551eb6b8e049
COVID-468.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-468.png,COVID/images/COVID-468.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-468.png,COVID/masks/COVID-468.png,True,False,299,299,L,157.52,5170fab91db0b77ddf8c010b5eaa7e62
COVID-469.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-469.png,COVID/images/COVID-469.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-469.png,COVID/masks/COVID-469.png,True,False,299,299,L,146.9,f85ee8eb68993403eb081d9e127d26c1
COVID-47.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-47.png,COVID/images/COVID-47.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-47.png,COVID/masks/COVID-47.png,True,False,299,299,L,142.37,e1cf625408ddb6d50f36beee61c8b622
COVID-470.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-470.png,COVID/images/COVID-470.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-470.png,COVID/masks/COVID-470.png,True,False,299,299,L,145.01,580d3a1ee689025b8c956a97ac79a16c
COVID-471.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-471.png,COVID/images/COVID-471.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-471.png,COVID/masks/COVID-471.png,True,False,299,299,L,127.61,31aa59a3ff9ea24a39b6341a58fef59e
COVID-472.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-472.png,COVID/images/COVID-472.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-472.png,COVID/masks/COVID-472.png,True,False,299,299,L,107.33,25cf4af98a63379af2e75b8715cc364c
COVID-473.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-473.png,COVID/images/COVID-473.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-473.png,COVID/masks/COVID-473.png,True,False,299,299,L,134.32,3ec3948746a04fa3a9b2bad17c8c86c0
COVID-474.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-474.png,COVID/images/COVID-474.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-474.png,COVID/masks/COVID-474.png,True,False,299,299,L,158.42,d5df895aeafe45dfbc78cea33df2ddca
COVID-475.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-475.png,COVID/images/COVID-475.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-475.png,COVID/masks/COVID-475.png,True,False,299,299,L,114.51,8fee7fb2c72cb6d2dd54bbf215abb0ec
COVID-476.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-476.png,COVID/images/COVID-476.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-476.png,COVID/masks/COVID-476.png,True,False,299,299,L,149.94,38451729f132b310b820a5da4b425344
COVID-477.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-477.png,COVID/images/COVID-477.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-477.png,COVID/masks/COVID-477.png,True,False,299,299,L,148.97,94bc5bd54165076dc6b97bb005edcec8
COVID-478.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-478.png,COVID/images/COVID-478.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-478.png,COVID/masks/COVID-478.png,True,False,299,299,L,114.77,d5e59f62cd527fdf1ab43ac4e5a3b186
COVID-479.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-479.png,COVID/images/COVID-479.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-479.png,COVID/masks/COVID-479.png,True,False,299,299,L,112.24,63cd04e485008c1093f4a08fe4093fd3
COVID-48.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-48.png,COVID/images/COVID-48.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-48.png,COVID/masks/COVID-48.png,True,False,299,299,L,157.05,d486dcec1d88a9522403c5e5eeab3c25
COVID-480.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-480.png,COVID/images/COVID-480.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-480.png,COVID/masks/COVID-480.png,True,False,299,299,L,149.55,9c709dd31aef3f72eb101efe93383771
COVID-481.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-481.png,COVID/images/COVID-481.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-481.png,COVID/masks/COVID-481.png,True,False,299,299,L,46.62,873edbd06ea9a5a16369cb6a8bb3818b
COVID-482.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-482.png,COVID/images/COVID-482.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-482.png,COVID/masks/COVID-482.png,True,False,299,299,L,121.36,5a510356146387331a770ccb17379f98
COVID-483.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-483.png,COVID/images/COVID-483.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-483.png,COVID/masks/COVID-483.png,True,False,299,299,L,150.73,cfe990870684862a96b73b8087192efd
COVID-484.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-484.png,COVID/images/COVID-484.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-484.png,COVID/masks/COVID-484.png,True,False,299,299,L,117.01,6247fc10b0625a3a20be3a16cdb21041
COVID-485.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-485.png,COVID/images/COVID-485.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-485.png,COVID/masks/COVID-485.png,True,False,299,299,L,154.18,c7de00843995a64eda8bf2fa4647fac6
COVID-486.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-486.png,COVID/images/COVID-486.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-486.png,COVID/masks/COVID-486.png,True,False,299,299,L,148.64,af2243698dc32521f455418485d82ef5
COVID-487.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-487.png,COVID/images/COVID-487.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-487.png,COVID/masks/COVID-487.png,True,False,299,299,L,153.24,02c75a7e92e2410e9d814a03a67380d9
COVID-488.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-488.png,COVID/images/COVID-488.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-488.png,COVID/masks/COVID-488.png,True,False,299,299,L,132.55,4662d062d76b4fc96e522927edb53aa1
COVID-489.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-489.png,COVID/images/COVID-489.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-489.png,COVID/masks/COVID-489.png,True,False,299,299,L,140.62,a4ca6a4f8dd8e09fe23aba3038089673
COVID-49.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-49.png,COVID/images/COVID-49.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-49.png,COVID/masks/COVID-49.png,True,False,299,299,L,178.22,e0e0a347b19729c214fd71221f8401ab
COVID-490.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-490.png,COVID/images/COVID-490.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-490.png,COVID/masks/COVID-490.png,True,False,299,299,L,154.05,73582ca5ef5d71dcfd600284a7124026
COVID-491.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-491.png,COVID/images/COVID-491.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-491.png,COVID/masks/COVID-491.png,True,False,299,299,L,107.63,cc9e947a1f1d3e5ed9e8abdf67fd2944
COVID-492.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-492.png,COVID/images/COVID-492.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-492.png,COVID/masks/COVID-492.png,True,False,299,299,L,156.34,51f9fdbd8443d1ceade0fd5b336bc267
COVID-493.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-493.png,COVID/images/COVID-493.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-493.png,COVID/masks/COVID-493.png,True,False,299,299,L,161.23,72574301dd5089d06dee58d3ed059fbb
COVID-494.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-494.png,COVID/images/COVID-494.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-494.png,COVID/masks/COVID-494.png,True,False,299,299,L,146.57,2b51e4d7f229a9bafe6f12ed04a381a1
COVID-495.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-495.png,COVID/images/COVID-495.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-495.png,COVID/masks/COVID-495.png,True,False,299,299,L,186.65,ff3e5ee52a6d068e237d44c9ee086b02
COVID-496.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-496.png,COVID/images/COVID-496.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-496.png,COVID/masks/COVID-496.png,True,False,299,299,L,162.01,732b567c10c4e461cf10ede52be3768e
COVID-497.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-497.png,COVID/images/COVID-497.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-497.png,COVID/masks/COVID-497.png,True,False,299,299,L,147.59,1010a049787002d785fe97b06d6b50c1
COVID-498.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-498.png,COVID/images/COVID-498.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-498.png,COVID/masks/COVID-498.png,True,False,299,299,L,162.51,73f9f00bf39395a885eb03868cd4b934
COVID-499.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-499.png,COVID/images/COVID-499.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-499.png,COVID/masks/COVID-499.png,True,False,299,299,L,164.41,a5925797d236fb702462f4b132f22f55
COVID-5.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-5.png,COVID/images/COVID-5.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-5.png,COVID/masks/COVID-5.png,True,False,299,299,L,167.84,671a649204be2bf32d1beeab230fd662
COVID-50.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-50.png,COVID/images/COVID-50.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-50.png,COVID/masks/COVID-50.png,True,False,299,299,L,145.4,f5208eafd23b90e1a7ac05d25d0d7293
COVID-500.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-500.png,COVID/images/COVID-500.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-500.png,COVID/masks/COVID-500.png,True,False,299,299,L,153.0,70113bbbddfdf54f3e2b4ff004b62ccc
COVID-501.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-501.png,COVID/images/COVID-501.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-501.png,COVID/masks/COVID-501.png,True,False,299,299,L,150.55,9362cc86c3cedc19439f7c3a0cb49c04
COVID-502.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-502.png,COVID/images/COVID-502.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-502.png,COVID/masks/COVID-502.png,True,False,299,299,L,185.25,a9a82044702e060bb4425edf0fa2837a
COVID-503.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-503.png,COVID/images/COVID-503.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-503.png,COVID/masks/COVID-503.png,True,False,299,299,L,149.68,04c8bbda789164ffcd02ca1455f70693
COVID-504.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-504.png,COVID/images/COVID-504.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-504.png,COVID/masks/COVID-504.png,True,False,299,299,L,153.92,fc6bce094d2af346d1ded11bdc433c71
COVID-505.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-505.png,COVID/images/COVID-505.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-505.png,COVID/masks/COVID-505.png,True,False,299,299,L,113.35,19c3d2a4f739601abeb81e5b9bece4d6
COVID-506.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-506.png,COVID/images/COVID-506.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-506.png,COVID/masks/COVID-506.png,True,False,299,299,L,163.13,49229ada086ce2c4f0f6230d13a60c35
COVID-507.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-507.png,COVID/images/COVID-507.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-507.png,COVID/masks/COVID-507.png,True,False,299,299,L,167.37,8b7a9a31bf68f462d0824757b9e89e7c
COVID-508.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-508.png,COVID/images/COVID-508.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-508.png,COVID/masks/COVID-508.png,True,False,299,299,L,121.68,9bf1527ae411ca1bcd10a45b0c20b023
COVID-509.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-509.png,COVID/images/COVID-509.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-509.png,COVID/masks/COVID-509.png,True,False,299,299,L,155.97,8e6bfbb4f4ba1bb32096432e3396ec04
COVID-51.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-51.png,COVID/images/COVID-51.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-51.png,COVID/masks/COVID-51.png,True,False,299,299,L,172.56,7ae1cc5f8aec9c2e6635c63e6b549698
COVID-510.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-510.png,COVID/images/COVID-510.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-510.png,COVID/masks/COVID-510.png,True,False,299,299,L,183.9,640fa9a31f5f3e6ae9be85a77bfeda39
COVID-511.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-511.png,COVID/images/COVID-511.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-511.png,COVID/masks/COVID-511.png,True,False,299,299,L,150.25,603146dad1d81270ac1b62a4723b3fe5
COVID-512.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-512.png,COVID/images/COVID-512.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-512.png,COVID/masks/COVID-512.png,True,False,299,299,L,155.68,e347029f8d1651a021cbf2f7a3c04210
COVID-513.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-513.png,COVID/images/COVID-513.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-513.png,COVID/masks/COVID-513.png,True,False,299,299,L,136.25,cc0ce3ee61223b453a7ab65e8e447bfd
COVID-514.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-514.png,COVID/images/COVID-514.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-514.png,COVID/masks/COVID-514.png,True,False,299,299,L,152.56,5e118c8d5de4661963746fbb72329cf4
COVID-515.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-515.png,COVID/images/COVID-515.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-515.png,COVID/masks/COVID-515.png,True,False,299,299,L,137.45,e8ce2ea9a0b4d3dcdaad5a5041f1a096
COVID-516.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-516.png,COVID/images/COVID-516.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-516.png,COVID/masks/COVID-516.png,True,False,299,299,L,172.22,0de463ede19ebbc2554b745ef9997ec4
COVID-517.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-517.png,COVID/images/COVID-517.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-517.png,COVID/masks/COVID-517.png,True,False,299,299,L,121.72,cedca9c81f7f43849262556103c7b5d5
COVID-518.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-518.png,COVID/images/COVID-518.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-518.png,COVID/masks/COVID-518.png,True,False,299,299,L,158.99,0212d81f1f2ed495feee98b603c46605
COVID-519.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-519.png,COVID/images/COVID-519.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-519.png,COVID/masks/COVID-519.png,True,False,299,299,L,132.85,2f20a6057b823f8a6d9a45a3184f6cd6
COVID-52.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-52.png,COVID/images/COVID-52.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-52.png,COVID/masks/COVID-52.png,True,False,299,299,L,178.22,e0e0a347b19729c214fd71221f8401ab
COVID-520.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-520.png,COVID/images/COVID-520.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-520.png,COVID/masks/COVID-520.png,True,False,299,299,L,134.78,8f95c29ccc15f1957a6d19386bc5a7eb
COVID-521.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-521.png,COVID/images/COVID-521.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-521.png,COVID/masks/COVID-521.png,True,False,299,299,L,105.26,46171cf63976ce845265899522c9e0bd
COVID-522.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-522.png,COVID/images/COVID-522.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-522.png,COVID/masks/COVID-522.png,True,False,299,299,L,94.42,317e02378c1683c5ce550faa85487b51
COVID-523.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-523.png,COVID/images/COVID-523.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-523.png,COVID/masks/COVID-523.png,True,False,299,299,L,68.14,2721c649ad255bb47a1b8a0b0383859b
COVID-524.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-524.png,COVID/images/COVID-524.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-524.png,COVID/masks/COVID-524.png,True,False,299,299,L,95.27,10dba9666139a5716e80d7e54e0fec59
COVID-525.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-525.png,COVID/images/COVID-525.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-525.png,COVID/masks/COVID-525.png,True,False,299,299,L,119.03,0e2c0880cb7bc7dd16cd9b44bd883c87
COVID-526.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-526.png,COVID/images/COVID-526.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-526.png,COVID/masks/COVID-526.png,True,False,299,299,L,125.3,e0137e6273c9c94f6ab6227fe566c62d
COVID-527.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-527.png,COVID/images/COVID-527.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-527.png,COVID/masks/COVID-527.png,True,False,299,299,L,98.93,831ece52deaefeccbc14c8b688b618f2
COVID-528.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-528.png,COVID/images/COVID-528.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-528.png,COVID/masks/COVID-528.png,True,False,299,299,L,168.64,3a58b562841ae1b0ae5f210ac2da6c2b
COVID-529.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-529.png,COVID/images/COVID-529.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-529.png,COVID/masks/COVID-529.png,True,False,299,299,L,96.98,6222d4d2ccf407526d21e5b0cb481141
COVID-53.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-53.png,COVID/images/COVID-53.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-53.png,COVID/masks/COVID-53.png,True,False,299,299,L,147.17,9696dce9632de088aa049958f95b300f
COVID-530.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-530.png,COVID/images/COVID-530.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-530.png,COVID/masks/COVID-530.png,True,False,299,299,L,160.59,b972f70ff84f1802952aefa4556a5fb2
COVID-531.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-531.png,COVID/images/COVID-531.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-531.png,COVID/masks/COVID-531.png,True,False,299,299,L,167.91,5f70b8ce8fc3e34339f2e51d2ca389df
COVID-532.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-532.png,COVID/images/COVID-532.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-532.png,COVID/masks/COVID-532.png,True,False,299,299,L,141.46,fccf9ee9eaada5f6a26614c9d79473c5
COVID-533.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-533.png,COVID/images/COVID-533.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-533.png,COVID/masks/COVID-533.png,True,False,299,299,L,126.21,af8ebde143f2d9f067a4fd479b3ebba0
COVID-534.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-534.png,COVID/images/COVID-534.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-534.png,COVID/masks/COVID-534.png,True,False,299,299,L,165.73,502bd576c0a5e9f94bac5144fe8c207b
COVID-535.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-535.png,COVID/images/COVID-535.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-535.png,COVID/masks/COVID-535.png,True,False,299,299,L,102.9,821fdcf81aa6375312a87892d9f40a70
COVID-536.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-536.png,COVID/images/COVID-536.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-536.png,COVID/masks/COVID-536.png,True,False,299,299,L,155.09,b3de442fc763f13e3bae7abdf83e563d
COVID-537.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-537.png,COVID/images/COVID-537.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-537.png,COVID/masks/COVID-537.png,True,False,299,299,L,126.5,f79d133985d267b46a6d956a64a3c1d0
COVID-538.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-538.png,COVID/images/COVID-538.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-538.png,COVID/masks/COVID-538.png,True,False,299,299,L,111.79,819fe78133b1d09227da719b0c28e583
COVID-539.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-539.png,COVID/images/COVID-539.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-539.png,COVID/masks/COVID-539.png,True,False,299,299,L,152.04,07798fdfcbf8296aaee8ddef20924d8c
COVID-54.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-54.png,COVID/images/COVID-54.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-54.png,COVID/masks/COVID-54.png,True,False,299,299,L,126.23,8bb0c169aba08f1653bfd7f3c09f2d18
COVID-540.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-540.png,COVID/images/COVID-540.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-540.png,COVID/masks/COVID-540.png,True,False,299,299,L,152.76,3a484e15a40c2bfd08dedb71b37d19eb
COVID-541.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-541.png,COVID/images/COVID-541.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-541.png,COVID/masks/COVID-541.png,True,False,299,299,L,111.41,6add187a0a1c6b84b2f53d0a471dd441
COVID-542.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-542.png,COVID/images/COVID-542.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-542.png,COVID/masks/COVID-542.png,True,False,299,299,L,136.94,3d44c48ac197e33d46ca83d3b83566d2
COVID-543.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-543.png,COVID/images/COVID-543.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-543.png,COVID/masks/COVID-543.png,True,False,299,299,L,129.61,882c1c0e38177ef5b65c882708b822a1
COVID-544.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-544.png,COVID/images/COVID-544.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-544.png,COVID/masks/COVID-544.png,True,False,299,299,L,152.75,ded6c20925eeb6c4a808dc7809b03d04
COVID-545.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-545.png,COVID/images/COVID-545.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-545.png,COVID/masks/COVID-545.png,True,False,299,299,L,108.64,b263796ad3490f9b753ea79039edb248
COVID-546.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-546.png,COVID/images/COVID-546.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-546.png,COVID/masks/COVID-546.png,True,False,299,299,L,137.42,401113e15887dc538bb753a93ffe6b6d
COVID-547.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-547.png,COVID/images/COVID-547.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-547.png,COVID/masks/COVID-547.png,True,False,299,299,L,168.7,4168e165d3ea4758a5d2efc1a015f99b
COVID-548.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-548.png,COVID/images/COVID-548.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-548.png,COVID/masks/COVID-548.png,True,False,299,299,L,134.44,6633dd65005837917e69febdc40d8b74
COVID-549.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-549.png,COVID/images/COVID-549.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-549.png,COVID/masks/COVID-549.png,True,False,299,299,L,155.97,e282dd2dec50f1d752258f64400c173a
COVID-55.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-55.png,COVID/images/COVID-55.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-55.png,COVID/masks/COVID-55.png,True,False,299,299,L,161.77,7189bdcd7e957e936fb8e5101693e621
COVID-550.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-550.png,COVID/images/COVID-550.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-550.png,COVID/masks/COVID-550.png,True,False,299,299,L,153.51,9fbd9b53941d93990bf67835a8a4a0f0
COVID-551.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-551.png,COVID/images/COVID-551.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-551.png,COVID/masks/COVID-551.png,True,False,299,299,L,147.46,bae2f7c3add3473db04df67af765e787
COVID-552.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-552.png,COVID/images/COVID-552.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-552.png,COVID/masks/COVID-552.png,True,False,299,299,L,92.73,cd36090a2bd979b3bb3323ad3e6faea1
COVID-553.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-553.png,COVID/images/COVID-553.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-553.png,COVID/masks/COVID-553.png,True,False,299,299,L,97.75,1eafa262d7a3ed51246d53e0335bb514
COVID-554.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-554.png,COVID/images/COVID-554.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-554.png,COVID/masks/COVID-554.png,True,False,299,299,L,140.88,4638530666225cb96d9bc2e9ad11f8a7
COVID-555.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-555.png,COVID/images/COVID-555.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-555.png,COVID/masks/COVID-555.png,True,False,299,299,L,113.68,6a2ea9b6511e803a31b2ba048043e907
COVID-556.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-556.png,COVID/images/COVID-556.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-556.png,COVID/masks/COVID-556.png,True,False,299,299,L,89.83,eb2c02e372d5ef9b5124740c4f965263
COVID-557.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-557.png,COVID/images/COVID-557.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-557.png,COVID/masks/COVID-557.png,True,False,299,299,L,160.04,4d4a54439a70f6d8176c9d1e7941ae20
COVID-558.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-558.png,COVID/images/COVID-558.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-558.png,COVID/masks/COVID-558.png,True,False,299,299,L,137.09,268351ec00a51db071e09df84b5c93cd
COVID-559.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-559.png,COVID/images/COVID-559.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-559.png,COVID/masks/COVID-559.png,True,False,299,299,L,167.65,66debf08746c616943b40e3e0bce1121
COVID-56.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-56.png,COVID/images/COVID-56.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-56.png,COVID/masks/COVID-56.png,True,False,299,299,L,121.82,37e70afe1250154c8e09737f67d124ae
COVID-560.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-560.png,COVID/images/COVID-560.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-560.png,COVID/masks/COVID-560.png,True,False,299,299,L,134.57,8805f1259c12bf4fd83a8f51c5b8650c
COVID-561.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-561.png,COVID/images/COVID-561.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-561.png,COVID/masks/COVID-561.png,True,False,299,299,L,126.97,cac9f97e864e74af7cca714bb38e16c0
COVID-562.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-562.png,COVID/images/COVID-562.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-562.png,COVID/masks/COVID-562.png,True,False,299,299,L,141.3,331ec78c1229bff9fb60298441e9466f
COVID-563.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-563.png,COVID/images/COVID-563.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-563.png,COVID/masks/COVID-563.png,True,False,299,299,L,115.02,2515c7b494228feb4c8591799d594ca3
COVID-564.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-564.png,COVID/images/COVID-564.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-564.png,COVID/masks/COVID-564.png,True,False,299,299,L,140.21,17f1a348a40c2a42735d21fa1bcbaba0
COVID-565.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-565.png,COVID/images/COVID-565.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-565.png,COVID/masks/COVID-565.png,True,False,299,299,L,150.41,52e439c9c12efcf315b1ef8e17c7190e
COVID-566.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-566.png,COVID/images/COVID-566.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-566.png,COVID/masks/COVID-566.png,True,False,299,299,L,146.79,bfeec07c0fbe7c94f1619394a0ef920d
COVID-567.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-567.png,COVID/images/COVID-567.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-567.png,COVID/masks/COVID-567.png,True,False,299,299,L,136.25,71bbf25fa872d7edd186b8f4b02559be
COVID-568.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-568.png,COVID/images/COVID-568.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-568.png,COVID/masks/COVID-568.png,True,False,299,299,L,163.8,7f58c47f3edb8eb40768e8d09867a47c
COVID-569.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-569.png,COVID/images/COVID-569.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-569.png,COVID/masks/COVID-569.png,True,False,299,299,L,117.98,0669f25bb80d13d47db8acbc6f54399d
COVID-57.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-57.png,COVID/images/COVID-57.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-57.png,COVID/masks/COVID-57.png,True,False,299,299,L,124.27,5a35b1a4c913b6abcfd1e998896b7a18
COVID-570.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-570.png,COVID/images/COVID-570.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-570.png,COVID/masks/COVID-570.png,True,False,299,299,L,124.23,3c1e7750ae7ab758e3570138e9be0268
COVID-571.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-571.png,COVID/images/COVID-571.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-571.png,COVID/masks/COVID-571.png,True,False,299,299,L,129.42,ebe488308933422fb012d27d73eed9eb
COVID-572.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-572.png,COVID/images/COVID-572.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-572.png,COVID/masks/COVID-572.png,True,False,299,299,L,122.86,ec6f6b7f2937a73a41a7239add4aa4c7
COVID-573.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-573.png,COVID/images/COVID-573.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-573.png,COVID/masks/COVID-573.png,True,False,299,299,L,135.13,752e06735ebdb71555a87456a0a7121d
COVID-574.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-574.png,COVID/images/COVID-574.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-574.png,COVID/masks/COVID-574.png,True,False,299,299,L,169.51,9eff077ae787f37c2b17b60528db3cd4
COVID-575.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-575.png,COVID/images/COVID-575.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-575.png,COVID/masks/COVID-575.png,True,False,299,299,L,118.08,f21443a5586013ca9b6d5a4f20947ba3
COVID-576.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-576.png,COVID/images/COVID-576.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-576.png,COVID/masks/COVID-576.png,True,False,299,299,L,139.36,805b8f6e0bd349279c58b60c9bc13de7
COVID-577.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-577.png,COVID/images/COVID-577.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-577.png,COVID/masks/COVID-577.png,True,False,299,299,L,191.93,92076bc6840e5b6be0d9210c5d6c52de
COVID-578.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-578.png,COVID/images/COVID-578.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-578.png,COVID/masks/COVID-578.png,True,False,299,299,L,121.46,5416db134f774e25f582eecfee269ef1
COVID-579.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-579.png,COVID/images/COVID-579.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-579.png,COVID/masks/COVID-579.png,True,False,299,299,L,141.33,50e062741dd621d6a3d92a1e78a95265
COVID-58.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-58.png,COVID/images/COVID-58.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-58.png,COVID/masks/COVID-58.png,True,False,299,299,L,141.2,4a9b8a0c73532832e98975ae7188215e
COVID-580.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-580.png,COVID/images/COVID-580.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-580.png,COVID/masks/COVID-580.png,True,False,299,299,L,154.81,6760d2d0cef15eddaa59d7f7741074f3
COVID-581.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-581.png,COVID/images/COVID-581.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-581.png,COVID/masks/COVID-581.png,True,False,299,299,L,87.76,2b6f9438a3ebaedc719a089ebe9a0b42
COVID-582.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-582.png,COVID/images/COVID-582.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-582.png,COVID/masks/COVID-582.png,True,False,299,299,L,111.31,a2819d2fbb76f20f4136ebe0d5eed804
COVID-583.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-583.png,COVID/images/COVID-583.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-583.png,COVID/masks/COVID-583.png,True,False,299,299,L,178.27,cc295c0f2517485fa30f9aed9927a32a
COVID-584.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-584.png,COVID/images/COVID-584.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-584.png,COVID/masks/COVID-584.png,True,False,299,299,L,111.84,c9998c5a02a394a7d8b56c00da4f53cc
COVID-585.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-585.png,COVID/images/COVID-585.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-585.png,COVID/masks/COVID-585.png,True,False,299,299,L,150.73,5b3c203f826e9106b4853848160e6abb
COVID-586.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-586.png,COVID/images/COVID-586.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-586.png,COVID/masks/COVID-586.png,True,False,299,299,L,146.04,b2a7cd971c42fbb63efbd8c4138b81e5
COVID-587.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-587.png,COVID/images/COVID-587.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-587.png,COVID/masks/COVID-587.png,True,False,299,299,L,163.38,17c37f4d86002303fc2ada64ed3d2f47
COVID-588.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-588.png,COVID/images/COVID-588.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-588.png,COVID/masks/COVID-588.png,True,False,299,299,L,121.35,04afb919e4278bbf9b04235fdc24dd7d
COVID-589.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-589.png,COVID/images/COVID-589.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-589.png,COVID/masks/COVID-589.png,True,False,299,299,L,156.7,420774c46afa4435626428c0bd692e08
COVID-59.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-59.png,COVID/images/COVID-59.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-59.png,COVID/masks/COVID-59.png,True,False,299,299,L,174.13,5b8bd52a662328148b3b21a7dba38397
COVID-590.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-590.png,COVID/images/COVID-590.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-590.png,COVID/masks/COVID-590.png,True,False,299,299,L,117.45,c9398da4354a9c5ceb259d058856a2af
COVID-591.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-591.png,COVID/images/COVID-591.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-591.png,COVID/masks/COVID-591.png,True,False,299,299,L,145.94,5adae78f6d6ce500776641ac63d14e53
COVID-592.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-592.png,COVID/images/COVID-592.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-592.png,COVID/masks/COVID-592.png,True,False,299,299,L,109.65,05f1b1f12ccacb6408baa93de7228a32
COVID-593.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-593.png,COVID/images/COVID-593.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-593.png,COVID/masks/COVID-593.png,True,False,299,299,L,163.19,c078e8d466f66f114ff1390f38f22e26
COVID-594.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-594.png,COVID/images/COVID-594.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-594.png,COVID/masks/COVID-594.png,True,False,299,299,L,137.94,2ef64ac393a40a670719e88c7a469586
COVID-595.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-595.png,COVID/images/COVID-595.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-595.png,COVID/masks/COVID-595.png,True,False,299,299,L,166.69,158535d5bd6c964a490f89be5e14fdc8
COVID-596.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-596.png,COVID/images/COVID-596.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-596.png,COVID/masks/COVID-596.png,True,False,299,299,L,136.44,2ea66a6ff047f9de019dfe690716af89
COVID-597.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-597.png,COVID/images/COVID-597.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-597.png,COVID/masks/COVID-597.png,True,False,299,299,L,151.11,cf37c8c57e83d9f7adc7a8c244747a75
COVID-598.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-598.png,COVID/images/COVID-598.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-598.png,COVID/masks/COVID-598.png,True,False,299,299,L,150.85,f1d377b8ee90e2c504ae4529715d37ac
COVID-599.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-599.png,COVID/images/COVID-599.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-599.png,COVID/masks/COVID-599.png,True,False,299,299,L,156.59,48f016476749b597b229313295747ce9
COVID-6.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-6.png,COVID/images/COVID-6.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-6.png,COVID/masks/COVID-6.png,True,False,299,299,L,152.1,e241b3cd501964715bade5353630ad2e
COVID-60.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-60.png,COVID/images/COVID-60.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-60.png,COVID/masks/COVID-60.png,True,False,299,299,L,154.69,3c43f7e56e7db9f98e38b709408fb8b8
COVID-600.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-600.png,COVID/images/COVID-600.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-600.png,COVID/masks/COVID-600.png,True,False,299,299,L,159.99,82a84d56ef6bea69a6177a45d5b3c2e1
COVID-601.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-601.png,COVID/images/COVID-601.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-601.png,COVID/masks/COVID-601.png,True,False,299,299,L,146.75,78e3fe40db70c976e4864314c5465303
COVID-602.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-602.png,COVID/images/COVID-602.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-602.png,COVID/masks/COVID-602.png,True,False,299,299,L,130.81,200ba85353400c0461a34faee762bc90
COVID-603.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-603.png,COVID/images/COVID-603.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-603.png,COVID/masks/COVID-603.png,True,False,299,299,L,157.35,3e999713cc9d5cef8e529aa6252506df
COVID-604.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-604.png,COVID/images/COVID-604.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-604.png,COVID/masks/COVID-604.png,True,False,299,299,L,61.47,5b7e2e67dcf8fbcfab13bf31b804d416
COVID-605.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-605.png,COVID/images/COVID-605.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-605.png,COVID/masks/COVID-605.png,True,False,299,299,L,150.03,89bb50d28fab13c6ae29433dcb92bbf9
COVID-606.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-606.png,COVID/images/COVID-606.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-606.png,COVID/masks/COVID-606.png,True,False,299,299,L,147.59,2e25482ba81ce86be583e79ea2b90143
COVID-607.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-607.png,COVID/images/COVID-607.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-607.png,COVID/masks/COVID-607.png,True,False,299,299,L,118.4,df275daf345213eb0d9efcd47de79a9f
COVID-608.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-608.png,COVID/images/COVID-608.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-608.png,COVID/masks/COVID-608.png,True,False,299,299,L,131.09,9fa9d0e824026210c55c32aad5246a95
COVID-609.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-609.png,COVID/images/COVID-609.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-609.png,COVID/masks/COVID-609.png,True,False,299,299,L,152.44,862568b72290957262322fa9c6c9af26
COVID-61.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-61.png,COVID/images/COVID-61.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-61.png,COVID/masks/COVID-61.png,True,False,299,299,L,124.03,e96502aaec89790fa87e6d6a326d71a9
COVID-610.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-610.png,COVID/images/COVID-610.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-610.png,COVID/masks/COVID-610.png,True,False,299,299,L,158.67,9453bdd1620e377ec2c6d27bebd3a20e
COVID-611.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-611.png,COVID/images/COVID-611.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-611.png,COVID/masks/COVID-611.png,True,False,299,299,L,136.63,a82f6dbcaaa5ad3ec307696775686731
COVID-612.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-612.png,COVID/images/COVID-612.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-612.png,COVID/masks/COVID-612.png,True,False,299,299,L,144.24,fe8c081cabd51944ad4da8f26ed145f6
COVID-613.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-613.png,COVID/images/COVID-613.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-613.png,COVID/masks/COVID-613.png,True,False,299,299,L,153.86,006f92c15ec31cfa36042982c0c0dddd
COVID-614.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-614.png,COVID/images/COVID-614.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-614.png,COVID/masks/COVID-614.png,True,False,299,299,L,167.58,890eb53e5026635e8ff272a7dbd9a7f0
COVID-615.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-615.png,COVID/images/COVID-615.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-615.png,COVID/masks/COVID-615.png,True,False,299,299,L,147.29,0ebd284b77c99122e0c53cdc1619a869
COVID-616.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-616.png,COVID/images/COVID-616.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-616.png,COVID/masks/COVID-616.png,True,False,299,299,L,149.64,3f393b114734dd5723059b38cf54d7dc
COVID-617.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-617.png,COVID/images/COVID-617.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-617.png,COVID/masks/COVID-617.png,True,False,299,299,L,151.51,87fc890479208cab63512ee3569a4dac
COVID-618.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-618.png,COVID/images/COVID-618.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-618.png,COVID/masks/COVID-618.png,True,False,299,299,L,147.27,47bb8def9b256302c505eb67aa14264e
COVID-619.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-619.png,COVID/images/COVID-619.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-619.png,COVID/masks/COVID-619.png,True,False,299,299,L,179.96,85d574829891927f42f7cf32b684a728
COVID-62.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-62.png,COVID/images/COVID-62.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-62.png,COVID/masks/COVID-62.png,True,False,299,299,L,138.45,ada8cf0fd69aea3430afa7808e016f77
COVID-620.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-620.png,COVID/images/COVID-620.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-620.png,COVID/masks/COVID-620.png,True,False,299,299,L,139.49,487a44e16827bd847709937994f420eb
COVID-621.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-621.png,COVID/images/COVID-621.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-621.png,COVID/masks/COVID-621.png,True,False,299,299,L,149.41,9057f6d45a97afc85e5f7b447c419541
COVID-622.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-622.png,COVID/images/COVID-622.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-622.png,COVID/masks/COVID-622.png,True,False,299,299,L,170.85,bfa0cb786a9afc79175a9c7a2769b2bf
COVID-623.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-623.png,COVID/images/COVID-623.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-623.png,COVID/masks/COVID-623.png,True,False,299,299,L,135.92,1bdd8a54482824f16a0e6bb63466dc96
COVID-624.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-624.png,COVID/images/COVID-624.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-624.png,COVID/masks/COVID-624.png,True,False,299,299,L,184.91,756f7e0d844debe1fdf4eb1030a717ae
COVID-625.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-625.png,COVID/images/COVID-625.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-625.png,COVID/masks/COVID-625.png,True,False,299,299,L,177.35,2b5aeba29998309765ff6d69df33d7e0
COVID-626.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-626.png,COVID/images/COVID-626.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-626.png,COVID/masks/COVID-626.png,True,False,299,299,L,97.16,1aca92148e4c9dfe73bc822a83cccca2
COVID-627.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-627.png,COVID/images/COVID-627.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-627.png,COVID/masks/COVID-627.png,True,False,299,299,L,165.69,94f8d2f04a349790fb735532048ca578
COVID-628.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-628.png,COVID/images/COVID-628.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-628.png,COVID/masks/COVID-628.png,True,False,299,299,L,160.4,9d2b2304ebcb12c05a39cd79d2c5183e
COVID-629.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-629.png,COVID/images/COVID-629.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-629.png,COVID/masks/COVID-629.png,True,False,299,299,L,119.53,a12a1b5c07acaf9b6304242326b03d59
COVID-63.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-63.png,COVID/images/COVID-63.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-63.png,COVID/masks/COVID-63.png,True,False,299,299,L,151.93,2faa9a8742833874b89f8c6a649db858
COVID-630.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-630.png,COVID/images/COVID-630.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-630.png,COVID/masks/COVID-630.png,True,False,299,299,L,161.37,6be6af0e398fafcb8c468b4ef7f8c0f8
COVID-631.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-631.png,COVID/images/COVID-631.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-631.png,COVID/masks/COVID-631.png,True,False,299,299,L,161.84,d6eee2fb873b9709717c2c5f81712474
COVID-632.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-632.png,COVID/images/COVID-632.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-632.png,COVID/masks/COVID-632.png,True,False,299,299,L,151.51,ac78876070d171a5f9b123ec6f5ebf47
COVID-633.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-633.png,COVID/images/COVID-633.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-633.png,COVID/masks/COVID-633.png,True,False,299,299,L,131.03,a6f3ba126623b133aa379ee3031cc65e
COVID-634.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-634.png,COVID/images/COVID-634.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-634.png,COVID/masks/COVID-634.png,True,False,299,299,L,164.22,08ea66561911b9e4b5d7ebad06b6109c
COVID-635.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-635.png,COVID/images/COVID-635.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-635.png,COVID/masks/COVID-635.png,True,False,299,299,L,101.84,b88ffca2513a860329583f6aea9af458
COVID-636.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-636.png,COVID/images/COVID-636.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-636.png,COVID/masks/COVID-636.png,True,False,299,299,L,170.6,e68e322eafe60a5c3a25ff68d8871da8
COVID-637.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-637.png,COVID/images/COVID-637.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-637.png,COVID/masks/COVID-637.png,True,False,299,299,L,121.19,a5bfd817c43b0b3ed878abe7dc8df833
COVID-638.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-638.png,COVID/images/COVID-638.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-638.png,COVID/masks/COVID-638.png,True,False,299,299,L,190.34,839923253dce090d1e33e44d5f683fb7
COVID-639.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-639.png,COVID/images/COVID-639.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-639.png,COVID/masks/COVID-639.png,True,False,299,299,L,178.07,f246d9cab1c23dd310fdee7f215b54c7
COVID-64.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-64.png,COVID/images/COVID-64.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-64.png,COVID/masks/COVID-64.png,True,False,299,299,L,144.03,9ce0bfa53efc936e14c1d3c888595aef
COVID-640.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-640.png,COVID/images/COVID-640.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-640.png,COVID/masks/COVID-640.png,True,False,299,299,L,169.83,a8a97099b8b17a58f6455e3069ae0d4c
COVID-641.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-641.png,COVID/images/COVID-641.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-641.png,COVID/masks/COVID-641.png,True,False,299,299,L,147.24,624e5fabfce6cb5d3922b01a63a67964
COVID-642.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-642.png,COVID/images/COVID-642.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-642.png,COVID/masks/COVID-642.png,True,False,299,299,L,139.88,63697e908302e98a7b83bc40e32c7894
COVID-643.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-643.png,COVID/images/COVID-643.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-643.png,COVID/masks/COVID-643.png,True,False,299,299,L,176.17,bdfc6695076c71ec5c67dcce231a9fe4
COVID-644.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-644.png,COVID/images/COVID-644.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-644.png,COVID/masks/COVID-644.png,True,False,299,299,L,121.18,ea43f868d36801cbabe573ca4201e580
COVID-645.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-645.png,COVID/images/COVID-645.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-645.png,COVID/masks/COVID-645.png,True,False,299,299,L,150.02,af2b96bb79b921de6dd367f701196f9c
COVID-646.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-646.png,COVID/images/COVID-646.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-646.png,COVID/masks/COVID-646.png,True,False,299,299,L,152.83,6e203120bb0a28278132664be67e3120
COVID-647.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-647.png,COVID/images/COVID-647.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-647.png,COVID/masks/COVID-647.png,True,False,299,299,L,152.29,89825414c791cfc35462797634e04645
COVID-648.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-648.png,COVID/images/COVID-648.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-648.png,COVID/masks/COVID-648.png,True,False,299,299,L,158.01,f08ecf001af4232c815056747b63d9fb
COVID-649.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-649.png,COVID/images/COVID-649.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-649.png,COVID/masks/COVID-649.png,True,False,299,299,L,102.01,814b3fb05be7bd7cb4a0ed537568f93a
COVID-65.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-65.png,COVID/images/COVID-65.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-65.png,COVID/masks/COVID-65.png,True,False,299,299,L,143.41,00a4909c5ebf258b236fba00443fc369
COVID-650.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-650.png,COVID/images/COVID-650.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-650.png,COVID/masks/COVID-650.png,True,False,299,299,L,157.96,80730b323165cad9c4467a81dbafb83a
COVID-651.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-651.png,COVID/images/COVID-651.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-651.png,COVID/masks/COVID-651.png,True,False,299,299,L,153.84,1c6bed45e51499e99f77bbc8a8cac601
COVID-652.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-652.png,COVID/images/COVID-652.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-652.png,COVID/masks/COVID-652.png,True,False,299,299,L,137.78,b8b143080b6e518bc5607640d96de87d
COVID-653.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-653.png,COVID/images/COVID-653.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-653.png,COVID/masks/COVID-653.png,True,False,299,299,L,187.93,23edb00e7bb24726213a4a66614948e9
COVID-654.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-654.png,COVID/images/COVID-654.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-654.png,COVID/masks/COVID-654.png,True,False,299,299,L,142.0,b13d8377dc91e99e5b9c8534184498dc
COVID-655.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-655.png,COVID/images/COVID-655.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-655.png,COVID/masks/COVID-655.png,True,False,299,299,L,99.48,a928d3bc8c543883389bddbebed0fe61
COVID-656.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-656.png,COVID/images/COVID-656.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-656.png,COVID/masks/COVID-656.png,True,False,299,299,L,192.25,26cc5b8c730714008852f24212b5c6bc
COVID-657.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-657.png,COVID/images/COVID-657.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-657.png,COVID/masks/COVID-657.png,True,False,299,299,L,158.82,c1bc8261badfd651b233633ecbf80c99
COVID-658.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-658.png,COVID/images/COVID-658.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-658.png,COVID/masks/COVID-658.png,True,False,299,299,L,137.27,6af4a08e24188d78a7d53d2718e6e066
COVID-659.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-659.png,COVID/images/COVID-659.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-659.png,COVID/masks/COVID-659.png,True,False,299,299,L,162.44,6c155e0ca5918896da11f057cc081b24
COVID-66.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-66.png,COVID/images/COVID-66.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-66.png,COVID/masks/COVID-66.png,True,False,299,299,L,156.41,355dcb0e8143cf63c94de9506f181234
COVID-660.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-660.png,COVID/images/COVID-660.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-660.png,COVID/masks/COVID-660.png,True,False,299,299,L,187.19,dbbc5213477ce2935642cf338139a032
COVID-661.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-661.png,COVID/images/COVID-661.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-661.png,COVID/masks/COVID-661.png,True,False,299,299,L,181.94,3681ae90940473e019fb52974034b99e
COVID-662.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-662.png,COVID/images/COVID-662.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-662.png,COVID/masks/COVID-662.png,True,False,299,299,L,148.18,616d503cec551c2e42861211c59d979d
COVID-663.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-663.png,COVID/images/COVID-663.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-663.png,COVID/masks/COVID-663.png,True,False,299,299,L,157.99,6cd80186cd2c4b62ae73127dd9808706
COVID-664.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-664.png,COVID/images/COVID-664.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-664.png,COVID/masks/COVID-664.png,True,False,299,299,L,155.51,6a9dfa838907818fa5b659bbda748795
COVID-665.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-665.png,COVID/images/COVID-665.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-665.png,COVID/masks/COVID-665.png,True,False,299,299,L,114.9,cab16080411261e1e2448e84649b1b64
COVID-666.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-666.png,COVID/images/COVID-666.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-666.png,COVID/masks/COVID-666.png,True,False,299,299,L,135.39,08328dfcbe5105c895477588833c2912
COVID-667.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-667.png,COVID/images/COVID-667.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-667.png,COVID/masks/COVID-667.png,True,False,299,299,L,157.01,bf8a506db45adff43be39a58bf8631fe
COVID-668.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-668.png,COVID/images/COVID-668.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-668.png,COVID/masks/COVID-668.png,True,False,299,299,L,138.9,cf894f486d6bc520c31fb12996f62818
COVID-669.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-669.png,COVID/images/COVID-669.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-669.png,COVID/masks/COVID-669.png,True,False,299,299,L,141.67,7cd716b2eb92cc89b1e4696b08cded84
COVID-67.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-67.png,COVID/images/COVID-67.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-67.png,COVID/masks/COVID-67.png,True,False,299,299,L,125.72,e7ff2831d0cb1fd208fcee2efe3b1f26
COVID-670.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-670.png,COVID/images/COVID-670.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-670.png,COVID/masks/COVID-670.png,True,False,299,299,L,158.67,e53cb6c460334a111b017ce0da62b24f
COVID-671.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-671.png,COVID/images/COVID-671.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-671.png,COVID/masks/COVID-671.png,True,False,299,299,L,163.12,9406e7338ae60c381acbaa485021e5a6
COVID-672.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-672.png,COVID/images/COVID-672.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-672.png,COVID/masks/COVID-672.png,True,False,299,299,L,100.68,4bf7bf3d8366defe8ae868224ad86a65
COVID-673.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-673.png,COVID/images/COVID-673.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-673.png,COVID/masks/COVID-673.png,True,False,299,299,L,152.89,2b3c6771b953fb065a3bebbb6c8b485d
COVID-674.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-674.png,COVID/images/COVID-674.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-674.png,COVID/masks/COVID-674.png,True,False,299,299,L,175.53,af41e7c3b388d8bd9c0e0e47cc0ecfd8
COVID-675.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-675.png,COVID/images/COVID-675.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-675.png,COVID/masks/COVID-675.png,True,False,299,299,L,102.41,1d72f5135c058e94731d207e78287832
COVID-676.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-676.png,COVID/images/COVID-676.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-676.png,COVID/masks/COVID-676.png,True,False,299,299,L,148.81,1d3a214a449d29098b6f4095231fb7c8
COVID-677.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-677.png,COVID/images/COVID-677.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-677.png,COVID/masks/COVID-677.png,True,False,299,299,L,182.74,578e6ec97ac888e43a49b914bee8f308
COVID-678.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-678.png,COVID/images/COVID-678.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-678.png,COVID/masks/COVID-678.png,True,False,299,299,L,116.49,6bf6c29ce3beeec5394ea753db876f13
COVID-679.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-679.png,COVID/images/COVID-679.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-679.png,COVID/masks/COVID-679.png,True,False,299,299,L,149.86,867d1f3086cf4b87c0bdbc9514f61d42
COVID-68.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-68.png,COVID/images/COVID-68.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-68.png,COVID/masks/COVID-68.png,True,False,299,299,L,131.25,4f147748b954c20949441d5ccec99c98
COVID-680.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-680.png,COVID/images/COVID-680.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-680.png,COVID/masks/COVID-680.png,True,False,299,299,L,156.35,92f473f2414240c0703011b05c47e79d
COVID-681.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-681.png,COVID/images/COVID-681.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-681.png,COVID/masks/COVID-681.png,True,False,299,299,L,110.33,0972fd414cf68f7b9a4f3321a6bd3164
COVID-682.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-682.png,COVID/images/COVID-682.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-682.png,COVID/masks/COVID-682.png,True,False,299,299,L,146.43,94dde04df265f96eb76d9c2a410d6b00
COVID-683.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-683.png,COVID/images/COVID-683.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-683.png,COVID/masks/COVID-683.png,True,False,299,299,L,184.31,e31dd828fcf21fbae276e77e848fe103
COVID-684.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-684.png,COVID/images/COVID-684.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-684.png,COVID/masks/COVID-684.png,True,False,299,299,L,146.4,3946f9b3baf6943a091c146994b1cce3
COVID-685.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-685.png,COVID/images/COVID-685.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-685.png,COVID/masks/COVID-685.png,True,False,299,299,L,150.29,74e1a2b8bcea533330cca989f5960f3f
COVID-686.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-686.png,COVID/images/COVID-686.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-686.png,COVID/masks/COVID-686.png,True,False,299,299,L,65.72,0424b777da5c8817bfd950bccfc7ea43
COVID-687.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-687.png,COVID/images/COVID-687.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-687.png,COVID/masks/COVID-687.png,True,False,299,299,L,111.06,6b6dd927e2ecc588fecb42f1b51f4f47
COVID-688.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-688.png,COVID/images/COVID-688.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-688.png,COVID/masks/COVID-688.png,True,False,299,299,L,143.68,92bd191de958737f99a071a4b97c1a99
COVID-689.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-689.png,COVID/images/COVID-689.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-689.png,COVID/masks/COVID-689.png,True,False,299,299,L,175.37,e325789e7dc4ab9ff86543a8e6da073e
COVID-69.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-69.png,COVID/images/COVID-69.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-69.png,COVID/masks/COVID-69.png,True,False,299,299,L,120.24,4a68ae08ceaabcd48b4fec84b677cb86
COVID-690.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-690.png,COVID/images/COVID-690.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-690.png,COVID/masks/COVID-690.png,True,False,299,299,L,158.71,4ac96769601de257c98b68a862651970
COVID-691.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-691.png,COVID/images/COVID-691.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-691.png,COVID/masks/COVID-691.png,True,False,299,299,L,164.45,91a54f8623ebce7d36802ffe204a31fc
COVID-692.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-692.png,COVID/images/COVID-692.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-692.png,COVID/masks/COVID-692.png,True,False,299,299,L,153.0,faac30015eaf312cceb352cff3a3fe61
COVID-693.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-693.png,COVID/images/COVID-693.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-693.png,COVID/masks/COVID-693.png,True,False,299,299,L,133.06,9d6e02606b9249a30f77b798e7cb8b52
COVID-694.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-694.png,COVID/images/COVID-694.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-694.png,COVID/masks/COVID-694.png,True,False,299,299,L,150.87,d846803e235deed27580d2e743667938
COVID-695.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-695.png,COVID/images/COVID-695.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-695.png,COVID/masks/COVID-695.png,True,False,299,299,L,106.92,9dcbad3dd9b99253b1ad40a48f20a585
COVID-696.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-696.png,COVID/images/COVID-696.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-696.png,COVID/masks/COVID-696.png,True,False,299,299,L,157.72,66bb5bab04c622aefd493522d63fc29f
COVID-697.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-697.png,COVID/images/COVID-697.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-697.png,COVID/masks/COVID-697.png,True,False,299,299,L,57.3,478386e239c567c8fe6aee8996c26007
COVID-698.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-698.png,COVID/images/COVID-698.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-698.png,COVID/masks/COVID-698.png,True,False,299,299,L,128.06,3bdebc22b0f03512fb35bd29067eecb0
COVID-699.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-699.png,COVID/images/COVID-699.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-699.png,COVID/masks/COVID-699.png,True,False,299,299,L,153.12,c99195a04dc376d94623d8b42be962c2
COVID-7.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-7.png,COVID/images/COVID-7.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-7.png,COVID/masks/COVID-7.png,True,False,299,299,L,149.37,f200fee537ca6c1752de8a653fa93e48
COVID-70.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-70.png,COVID/images/COVID-70.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-70.png,COVID/masks/COVID-70.png,True,False,299,299,L,145.09,3056a7668ac8b1ace1e1caf108eeac2f
COVID-700.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-700.png,COVID/images/COVID-700.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-700.png,COVID/masks/COVID-700.png,True,False,299,299,L,150.97,da04685f071ce0cb9d87b67c8f0b2a8d
COVID-701.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-701.png,COVID/images/COVID-701.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-701.png,COVID/masks/COVID-701.png,True,False,299,299,L,134.58,20ae0fb28566962d15077906b7a9e656
COVID-702.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-702.png,COVID/images/COVID-702.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-702.png,COVID/masks/COVID-702.png,True,False,299,299,L,98.55,ce26feeb2e3b4253ce2252bdd4d8ae36
COVID-703.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-703.png,COVID/images/COVID-703.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-703.png,COVID/masks/COVID-703.png,True,False,299,299,L,174.38,61ef538dd94065138a1a8903a018c2f2
COVID-704.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-704.png,COVID/images/COVID-704.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-704.png,COVID/masks/COVID-704.png,True,False,299,299,L,153.39,e19438d63c2f3798ba2a8fdc24dfdf7a
COVID-705.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-705.png,COVID/images/COVID-705.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-705.png,COVID/masks/COVID-705.png,True,False,299,299,L,125.94,75f77a7d1f80998a058cb2e671cb5004
COVID-706.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-706.png,COVID/images/COVID-706.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-706.png,COVID/masks/COVID-706.png,True,False,299,299,L,144.3,c3d884211f7dd4b6668c99c54496119a
COVID-707.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-707.png,COVID/images/COVID-707.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-707.png,COVID/masks/COVID-707.png,True,False,299,299,L,190.82,18d33e7ac786bb455e83ca546592841b
COVID-708.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-708.png,COVID/images/COVID-708.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-708.png,COVID/masks/COVID-708.png,True,False,299,299,L,105.1,1bb733ec50df4995194340b414f2ae78
COVID-709.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-709.png,COVID/images/COVID-709.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-709.png,COVID/masks/COVID-709.png,True,False,299,299,L,142.79,0c3090821bf54606c5c97357c060ed50
COVID-71.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-71.png,COVID/images/COVID-71.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-71.png,COVID/masks/COVID-71.png,True,False,299,299,L,141.93,15552e6d106c9498f451cf0e4f82a0ec
COVID-710.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-710.png,COVID/images/COVID-710.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-710.png,COVID/masks/COVID-710.png,True,False,299,299,L,180.31,de62c37c7d1042fa6693be01b377fac2
COVID-711.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-711.png,COVID/images/COVID-711.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-711.png,COVID/masks/COVID-711.png,True,False,299,299,L,160.32,b45ddb81279079c0789d109565626e0e
COVID-712.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-712.png,COVID/images/COVID-712.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-712.png,COVID/masks/COVID-712.png,True,False,299,299,L,150.22,8c96d78821e9a975bfc2bf1582632f8e
COVID-713.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-713.png,COVID/images/COVID-713.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-713.png,COVID/masks/COVID-713.png,True,False,299,299,L,162.7,e8dfe8fa30570ef4c644ea2c156f4a44
COVID-714.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-714.png,COVID/images/COVID-714.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-714.png,COVID/masks/COVID-714.png,True,False,299,299,L,132.78,8083f5b349490f2e6424ca1df7ff7bfe
COVID-715.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-715.png,COVID/images/COVID-715.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-715.png,COVID/masks/COVID-715.png,True,False,299,299,L,154.04,019459432c7936832c8ada0e04d829dc
COVID-716.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-716.png,COVID/images/COVID-716.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-716.png,COVID/masks/COVID-716.png,True,False,299,299,L,207.59,3b2a1d2348ac9e366039d815f24bbdfc
COVID-717.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-717.png,COVID/images/COVID-717.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-717.png,COVID/masks/COVID-717.png,True,False,299,299,L,143.56,eaff090964d2cb9505cfe4047dbf60ce
COVID-718.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-718.png,COVID/images/COVID-718.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-718.png,COVID/masks/COVID-718.png,True,False,299,299,L,154.37,04c82dad263d6faa93454ee6db8845d2
COVID-719.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-719.png,COVID/images/COVID-719.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-719.png,COVID/masks/COVID-719.png,True,False,299,299,L,176.92,f574efac7518b9db2a061fbe6634be61
COVID-72.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-72.png,COVID/images/COVID-72.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-72.png,COVID/masks/COVID-72.png,True,False,299,299,L,132.83,052a5157059e28c2352c30310880ff44
COVID-720.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-720.png,COVID/images/COVID-720.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-720.png,COVID/masks/COVID-720.png,True,False,299,299,L,175.34,97522045bbfe11b54908526d9ae12027
COVID-721.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-721.png,COVID/images/COVID-721.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-721.png,COVID/masks/COVID-721.png,True,False,299,299,L,180.08,13a1e7a0860ff7c61537f772553a4fa6
COVID-722.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-722.png,COVID/images/COVID-722.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-722.png,COVID/masks/COVID-722.png,True,False,299,299,L,121.46,de156cf8c9307f61cc96eaa8d258ccaa
COVID-723.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-723.png,COVID/images/COVID-723.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-723.png,COVID/masks/COVID-723.png,True,False,299,299,L,151.01,e4e6447d7b2de828bf9767017584b5f4
COVID-724.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-724.png,COVID/images/COVID-724.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-724.png,COVID/masks/COVID-724.png,True,False,299,299,L,100.96,9304748fb29aa5a36ca75aad1d1a57be
COVID-725.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-725.png,COVID/images/COVID-725.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-725.png,COVID/masks/COVID-725.png,True,False,299,299,L,165.57,8f1e158a20db3db8650295f1ddc21184
COVID-726.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-726.png,COVID/images/COVID-726.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-726.png,COVID/masks/COVID-726.png,True,False,299,299,L,186.79,ad919e5987fa4c58b1cb317d05d5c18f
COVID-727.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-727.png,COVID/images/COVID-727.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-727.png,COVID/masks/COVID-727.png,True,False,299,299,L,121.14,c175335788c4e39d4ea4c44cf444c942
COVID-728.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-728.png,COVID/images/COVID-728.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-728.png,COVID/masks/COVID-728.png,True,False,299,299,L,165.31,0447652245f0171534c6087d96d23779
COVID-729.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-729.png,COVID/images/COVID-729.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-729.png,COVID/masks/COVID-729.png,True,False,299,299,L,146.67,0ce3258f81503d1524af62f62a08bda0
COVID-73.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-73.png,COVID/images/COVID-73.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-73.png,COVID/masks/COVID-73.png,True,False,299,299,L,139.02,e558cd98cceddd72e5fb596ad650d5be
COVID-730.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-730.png,COVID/images/COVID-730.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-730.png,COVID/masks/COVID-730.png,True,False,299,299,L,141.08,2f843c70041c6eed7e5dd90ff286a984
COVID-731.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-731.png,COVID/images/COVID-731.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-731.png,COVID/masks/COVID-731.png,True,False,299,299,L,146.64,7b775f3d46f6262f300e1ec5c3d254ad
COVID-732.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-732.png,COVID/images/COVID-732.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-732.png,COVID/masks/COVID-732.png,True,False,299,299,L,126.93,f868c3378f5743c43b49cbcacd9f750e
COVID-733.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-733.png,COVID/images/COVID-733.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-733.png,COVID/masks/COVID-733.png,True,False,299,299,L,155.81,48e7de46eb260a7dc8d97692908e2914
COVID-734.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-734.png,COVID/images/COVID-734.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-734.png,COVID/masks/COVID-734.png,True,False,299,299,L,140.81,ff6e51a1f065c55c11e25ab9c05383bb
COVID-735.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-735.png,COVID/images/COVID-735.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-735.png,COVID/masks/COVID-735.png,True,False,299,299,L,127.07,90d616c688773bcc31b4dcbaaec57da8
COVID-736.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-736.png,COVID/images/COVID-736.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-736.png,COVID/masks/COVID-736.png,True,False,299,299,L,150.57,5a0abd23c3d4036332237d64caceeb68
COVID-737.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-737.png,COVID/images/COVID-737.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-737.png,COVID/masks/COVID-737.png,True,False,299,299,L,162.33,f4b349786f4674fa83611267811cd7aa
COVID-738.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-738.png,COVID/images/COVID-738.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-738.png,COVID/masks/COVID-738.png,True,False,299,299,L,154.33,9f217f4217a09cc588c1a4b717adfc7c
COVID-739.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-739.png,COVID/images/COVID-739.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-739.png,COVID/masks/COVID-739.png,True,False,299,299,L,179.49,813055c761b38044ae8aa4f3104e0a2a
COVID-74.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-74.png,COVID/images/COVID-74.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-74.png,COVID/masks/COVID-74.png,True,False,299,299,L,149.44,bf94eb38e310d1b7a8e1ff7016f33d89
COVID-740.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-740.png,COVID/images/COVID-740.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-740.png,COVID/masks/COVID-740.png,True,False,299,299,L,145.04,bb6b8ebb8ef5b4f485f687eb6bb3a46a
COVID-741.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-741.png,COVID/images/COVID-741.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-741.png,COVID/masks/COVID-741.png,True,False,299,299,L,77.68,b928e3eac6e749979e7323705f2a46e0
COVID-742.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-742.png,COVID/images/COVID-742.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-742.png,COVID/masks/COVID-742.png,True,False,299,299,L,160.68,7415a2d820f6f2d365f4b28db6729805
COVID-743.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-743.png,COVID/images/COVID-743.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-743.png,COVID/masks/COVID-743.png,True,False,299,299,L,158.38,c5d68e3b4bfe15fc0134d2d62ea5c529
COVID-744.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-744.png,COVID/images/COVID-744.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-744.png,COVID/masks/COVID-744.png,True,False,299,299,L,91.4,bb8ad9e32b89eeb0ec9fb86e12837bb6
COVID-745.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-745.png,COVID/images/COVID-745.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-745.png,COVID/masks/COVID-745.png,True,False,299,299,L,149.88,ec931b9dc1b99a9f6bbb9180207f3981
COVID-746.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-746.png,COVID/images/COVID-746.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-746.png,COVID/masks/COVID-746.png,True,False,299,299,L,148.43,9f81cdd86961737f249f384f91db0568
COVID-747.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-747.png,COVID/images/COVID-747.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-747.png,COVID/masks/COVID-747.png,True,False,299,299,L,156.42,998715ff2bbe838e65ac0f5e36524a82
COVID-748.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-748.png,COVID/images/COVID-748.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-748.png,COVID/masks/COVID-748.png,True,False,299,299,L,109.59,804b1f041e3ba1c69a880df0ffdea55b
COVID-749.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-749.png,COVID/images/COVID-749.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-749.png,COVID/masks/COVID-749.png,True,False,299,299,L,105.53,ca081dd56c504a8ec2072b84d1a65255
COVID-75.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-75.png,COVID/images/COVID-75.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-75.png,COVID/masks/COVID-75.png,True,False,299,299,L,141.25,d0922b6d31ffb84366c95de2f7a95126
COVID-750.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-750.png,COVID/images/COVID-750.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-750.png,COVID/masks/COVID-750.png,True,False,299,299,L,152.27,7304bf20b6e28465f3bf273c29c1b783
COVID-751.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-751.png,COVID/images/COVID-751.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-751.png,COVID/masks/COVID-751.png,True,False,299,299,L,155.1,65a6920dd35ed90bd2a237b1c9950ac3
COVID-752.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-752.png,COVID/images/COVID-752.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-752.png,COVID/masks/COVID-752.png,True,False,299,299,L,134.84,522e423d2fc3be7d2303d5ab83625ad6
COVID-753.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-753.png,COVID/images/COVID-753.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-753.png,COVID/masks/COVID-753.png,True,False,299,299,L,149.67,24c85e44fb3fd0119ee243f4ce03ed7c
COVID-754.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-754.png,COVID/images/COVID-754.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-754.png,COVID/masks/COVID-754.png,True,False,299,299,L,149.14,b0161ddd4f9d61f367920d7a0b2ba75b
COVID-755.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-755.png,COVID/images/COVID-755.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-755.png,COVID/masks/COVID-755.png,True,False,299,299,L,153.0,626a33f5e5cf6ab23e0e8c84759a0ca7
COVID-756.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-756.png,COVID/images/COVID-756.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-756.png,COVID/masks/COVID-756.png,True,False,299,299,L,145.74,232ea70423a2013f8b0545c07a075384
COVID-757.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-757.png,COVID/images/COVID-757.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-757.png,COVID/masks/COVID-757.png,True,False,299,299,L,147.95,4124232b4dde74a0412ebcf308f2c492
COVID-758.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-758.png,COVID/images/COVID-758.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-758.png,COVID/masks/COVID-758.png,True,False,299,299,L,142.78,5560fe2cd22da93ed37040cea8de7c75
COVID-759.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-759.png,COVID/images/COVID-759.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-759.png,COVID/masks/COVID-759.png,True,False,299,299,L,147.98,a6f6aad1a069c3bc8a0356e2ba74ebd7
COVID-76.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-76.png,COVID/images/COVID-76.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-76.png,COVID/masks/COVID-76.png,True,False,299,299,L,148.8,20d628d662ed7344b26f59d2397d25e8
COVID-760.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-760.png,COVID/images/COVID-760.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-760.png,COVID/masks/COVID-760.png,True,False,299,299,L,159.86,f118d154d0904acb921daac32ae4cdde
COVID-761.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-761.png,COVID/images/COVID-761.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-761.png,COVID/masks/COVID-761.png,True,False,299,299,L,152.8,2285ee1e3f6c70b81ef91c5faf43dbcf
COVID-762.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-762.png,COVID/images/COVID-762.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-762.png,COVID/masks/COVID-762.png,True,False,299,299,L,147.11,15c5683b886dab5dcd6326f4ccc9468e
COVID-763.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-763.png,COVID/images/COVID-763.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-763.png,COVID/masks/COVID-763.png,True,False,299,299,L,145.43,aff5e047e3b5149ff0b7b3b976142a0c
COVID-764.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-764.png,COVID/images/COVID-764.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-764.png,COVID/masks/COVID-764.png,True,False,299,299,L,147.23,d5527256be674ac58f8ee589616b5989
COVID-765.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-765.png,COVID/images/COVID-765.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-765.png,COVID/masks/COVID-765.png,True,False,299,299,L,135.3,cd285754d040adbc724d6820f832be2c
COVID-766.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-766.png,COVID/images/COVID-766.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-766.png,COVID/masks/COVID-766.png,True,False,299,299,L,146.61,5593eb8505e2734eb1e591aa9526a42c
COVID-767.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-767.png,COVID/images/COVID-767.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-767.png,COVID/masks/COVID-767.png,True,False,299,299,L,107.29,dd4c6c840efc8a8e1fa3cbd467f27849
COVID-768.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-768.png,COVID/images/COVID-768.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-768.png,COVID/masks/COVID-768.png,True,False,299,299,L,152.23,858437d62968f4eed5f6219d77aba7e3
COVID-769.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-769.png,COVID/images/COVID-769.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-769.png,COVID/masks/COVID-769.png,True,False,299,299,L,154.51,920eff06b23fe91a76f1a0e8a0d7b111
COVID-77.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-77.png,COVID/images/COVID-77.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-77.png,COVID/masks/COVID-77.png,True,False,299,299,L,185.38,df44588d06fbf4afd125bc1d12d7ebb5
COVID-770.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-770.png,COVID/images/COVID-770.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-770.png,COVID/masks/COVID-770.png,True,False,299,299,L,186.35,8ee9552c0a1e99b921e72d01a1116d12
COVID-771.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-771.png,COVID/images/COVID-771.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-771.png,COVID/masks/COVID-771.png,True,False,299,299,L,152.27,3c3e0d651ae97501c2857ef20d5526dd
COVID-772.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-772.png,COVID/images/COVID-772.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-772.png,COVID/masks/COVID-772.png,True,False,299,299,L,130.16,347f1248c6628578ac63a5862058e50a
COVID-773.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-773.png,COVID/images/COVID-773.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-773.png,COVID/masks/COVID-773.png,True,False,299,299,L,153.49,d829bdc1d0656118639afa3f6aa1c165
COVID-774.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-774.png,COVID/images/COVID-774.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-774.png,COVID/masks/COVID-774.png,True,False,299,299,L,158.85,a2bce1063a3804da73370c4b96e4ad24
COVID-775.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-775.png,COVID/images/COVID-775.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-775.png,COVID/masks/COVID-775.png,True,False,299,299,L,176.05,20b84e16ff79c6e4c430e60fdabef373
COVID-776.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-776.png,COVID/images/COVID-776.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-776.png,COVID/masks/COVID-776.png,True,False,299,299,L,155.55,ef5537fd1c8601ebb4e7516333b269bc
COVID-777.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-777.png,COVID/images/COVID-777.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-777.png,COVID/masks/COVID-777.png,True,False,299,299,L,130.04,d51464e248f846e812716bb8516bb182
COVID-778.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-778.png,COVID/images/COVID-778.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-778.png,COVID/masks/COVID-778.png,True,False,299,299,L,152.22,741ec9b0c3cb9ea345387c1684274097
COVID-779.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-779.png,COVID/images/COVID-779.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-779.png,COVID/masks/COVID-779.png,True,False,299,299,L,116.04,c663fe71945924cf515948f966efcd9a
COVID-78.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-78.png,COVID/images/COVID-78.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-78.png,COVID/masks/COVID-78.png,True,False,299,299,L,127.53,92c1504df975aed3dbea13be638a28fb
COVID-780.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-780.png,COVID/images/COVID-780.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-780.png,COVID/masks/COVID-780.png,True,False,299,299,L,161.71,fc35d8a4ed2298e6a80edd21f6bfd9c6
COVID-781.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-781.png,COVID/images/COVID-781.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-781.png,COVID/masks/COVID-781.png,True,False,299,299,L,100.62,bfeec5b6e01dbac44abd2d56bb1e2151
COVID-782.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-782.png,COVID/images/COVID-782.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-782.png,COVID/masks/COVID-782.png,True,False,299,299,L,126.04,978f7454dee3c85f27997b250f02025d
COVID-783.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-783.png,COVID/images/COVID-783.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-783.png,COVID/masks/COVID-783.png,True,False,299,299,L,146.54,4541dc41999bb3a68dca7e8fb12815c2
COVID-784.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-784.png,COVID/images/COVID-784.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-784.png,COVID/masks/COVID-784.png,True,False,299,299,L,155.09,18903ae5aff2c0465736af6ae301c293
COVID-785.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-785.png,COVID/images/COVID-785.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-785.png,COVID/masks/COVID-785.png,True,False,299,299,L,164.71,ca53e5635812861494901a8396c27d60
COVID-786.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-786.png,COVID/images/COVID-786.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-786.png,COVID/masks/COVID-786.png,True,False,299,299,L,159.02,86e3e601eff60476cc8a9bc5b062a7f5
COVID-787.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-787.png,COVID/images/COVID-787.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-787.png,COVID/masks/COVID-787.png,True,False,299,299,L,140.95,771916cce6b5efa715cef1169f516097
COVID-788.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-788.png,COVID/images/COVID-788.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-788.png,COVID/masks/COVID-788.png,True,False,299,299,L,169.6,601217d3c84bc621876920e367547fea
COVID-789.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-789.png,COVID/images/COVID-789.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-789.png,COVID/masks/COVID-789.png,True,False,299,299,L,158.7,f142411a943cc1c96233a87c1ac3379c
COVID-79.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-79.png,COVID/images/COVID-79.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-79.png,COVID/masks/COVID-79.png,True,False,299,299,L,165.14,d50213a7c5d24f6a6b04b1a04eec19ee
COVID-790.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-790.png,COVID/images/COVID-790.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-790.png,COVID/masks/COVID-790.png,True,False,299,299,L,138.8,553aa74f598f3dd5d671abdb785b623b
COVID-791.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-791.png,COVID/images/COVID-791.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-791.png,COVID/masks/COVID-791.png,True,False,299,299,L,130.21,470dcce4ba50743e872224352bd71de6
COVID-792.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-792.png,COVID/images/COVID-792.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-792.png,COVID/masks/COVID-792.png,True,False,299,299,L,152.16,346136eb90fabdf02c5629fbd5d33419
COVID-793.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-793.png,COVID/images/COVID-793.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-793.png,COVID/masks/COVID-793.png,True,False,299,299,L,148.1,d518e013f1c49741cb40589322bf62eb
COVID-794.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-794.png,COVID/images/COVID-794.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-794.png,COVID/masks/COVID-794.png,True,False,299,299,L,146.36,e0270fb143857b3a26ae4cd78f160117
COVID-795.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-795.png,COVID/images/COVID-795.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-795.png,COVID/masks/COVID-795.png,True,False,299,299,L,145.42,9261d024f4d20471d46b1acd7a09eb20
COVID-796.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-796.png,COVID/images/COVID-796.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-796.png,COVID/masks/COVID-796.png,True,False,299,299,L,132.9,647893d301d926c5ce5a94a92d2422f9
COVID-797.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-797.png,COVID/images/COVID-797.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-797.png,COVID/masks/COVID-797.png,True,False,299,299,L,139.21,15e4b6bcfa1d4ffee99981aef3617b3b
COVID-798.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-798.png,COVID/images/COVID-798.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-798.png,COVID/masks/COVID-798.png,True,False,299,299,L,163.76,8621e539b53d56eda32fdccef4541eba
COVID-799.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-799.png,COVID/images/COVID-799.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-799.png,COVID/masks/COVID-799.png,True,False,299,299,L,137.81,1dbf3087e1a59a97e7da61e72d169eb6
COVID-8.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-8.png,COVID/images/COVID-8.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-8.png,COVID/masks/COVID-8.png,True,False,299,299,L,160.48,ebdc3d131086d8927f50b56edacc864a
COVID-80.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-80.png,COVID/images/COVID-80.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-80.png,COVID/masks/COVID-80.png,True,False,299,299,L,152.73,414e72b5d4975554150670939cf3fe56
COVID-800.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-800.png,COVID/images/COVID-800.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-800.png,COVID/masks/COVID-800.png,True,False,299,299,L,148.77,ab582b48c01fe6ea5bbf9c29bfd63809
COVID-801.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-801.png,COVID/images/COVID-801.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-801.png,COVID/masks/COVID-801.png,True,False,299,299,L,168.62,6c725629c8bcce31ac347f2446ae46f0
COVID-802.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-802.png,COVID/images/COVID-802.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-802.png,COVID/masks/COVID-802.png,True,False,299,299,L,102.86,49583ea7002ae691b55b0e9b00f8698e
COVID-803.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-803.png,COVID/images/COVID-803.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-803.png,COVID/masks/COVID-803.png,True,False,299,299,L,142.63,deebb2f78f870e59342d0999f389b73b
COVID-804.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-804.png,COVID/images/COVID-804.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-804.png,COVID/masks/COVID-804.png,True,False,299,299,L,137.41,92119cd41b554a089ff5e9f57b29bd34
COVID-805.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-805.png,COVID/images/COVID-805.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-805.png,COVID/masks/COVID-805.png,True,False,299,299,L,162.49,6837b7b9ef8f915911f2ca6625d4e717
COVID-806.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-806.png,COVID/images/COVID-806.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-806.png,COVID/masks/COVID-806.png,True,False,299,299,L,155.05,65c313fc07c8354f5ecb7f7861dbae14
COVID-807.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-807.png,COVID/images/COVID-807.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-807.png,COVID/masks/COVID-807.png,True,False,299,299,L,149.62,1036946fb8ef833b38b461b0f135a963
COVID-808.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-808.png,COVID/images/COVID-808.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-808.png,COVID/masks/COVID-808.png,True,False,299,299,L,144.79,2d83287571f08f6fc26b94ab46a0d1a9
COVID-809.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-809.png,COVID/images/COVID-809.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-809.png,COVID/masks/COVID-809.png,True,False,299,299,L,152.95,e7f2ea60070b546e19160479e2653514
COVID-81.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-81.png,COVID/images/COVID-81.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-81.png,COVID/masks/COVID-81.png,True,False,299,299,L,146.27,0e86fe243e52794e93d6494b9a98ec65
COVID-810.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-810.png,COVID/images/COVID-810.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-810.png,COVID/masks/COVID-810.png,True,False,299,299,L,124.58,6aee0e3c78c51ff37cbd03ac7e46558d
COVID-811.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-811.png,COVID/images/COVID-811.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-811.png,COVID/masks/COVID-811.png,True,False,299,299,L,161.39,3f8943ddc8626c26a3e63b26aefc8bee
COVID-812.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-812.png,COVID/images/COVID-812.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-812.png,COVID/masks/COVID-812.png,True,False,299,299,L,131.89,70987d0764a16feb72ebe7de2bb2d254
COVID-813.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-813.png,COVID/images/COVID-813.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-813.png,COVID/masks/COVID-813.png,True,False,299,299,L,146.19,b8e17d8b30df460130634937ef812a48
COVID-814.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-814.png,COVID/images/COVID-814.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-814.png,COVID/masks/COVID-814.png,True,False,299,299,L,141.51,c53ea8ca2714540fb83151b6eb7f9b6b
COVID-815.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-815.png,COVID/images/COVID-815.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-815.png,COVID/masks/COVID-815.png,True,False,299,299,L,151.15,f0d960343d03f55ccb0946d7feae7fd1
COVID-816.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-816.png,COVID/images/COVID-816.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-816.png,COVID/masks/COVID-816.png,True,False,299,299,L,130.19,a965d789d81876c2343760bf63c487bd
COVID-817.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-817.png,COVID/images/COVID-817.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-817.png,COVID/masks/COVID-817.png,True,False,299,299,L,151.75,12061f9da4070da6fb99ab7542f659e3
COVID-818.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-818.png,COVID/images/COVID-818.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-818.png,COVID/masks/COVID-818.png,True,False,299,299,L,132.32,99c9a502989b7954f68069a78ad09833
COVID-819.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-819.png,COVID/images/COVID-819.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-819.png,COVID/masks/COVID-819.png,True,False,299,299,L,178.89,4ae336065e5c6824351d400d662f6cfd
COVID-82.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-82.png,COVID/images/COVID-82.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-82.png,COVID/masks/COVID-82.png,True,False,299,299,L,177.34,96f430aba09470ae50a82e61cbf75186
COVID-820.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-820.png,COVID/images/COVID-820.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-820.png,COVID/masks/COVID-820.png,True,False,299,299,L,119.67,2ef30a5f396b8fb9c06fa4936db7bf52
COVID-821.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-821.png,COVID/images/COVID-821.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-821.png,COVID/masks/COVID-821.png,True,False,299,299,L,141.25,be64883e67df794c7c0b9b32502f62c6
COVID-822.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-822.png,COVID/images/COVID-822.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-822.png,COVID/masks/COVID-822.png,True,False,299,299,L,132.96,f4ea315594e0fa4e5e3f34ff01ea4a76
COVID-823.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-823.png,COVID/images/COVID-823.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-823.png,COVID/masks/COVID-823.png,True,False,299,299,L,153.68,da4cf2f0ef5902c9253bdf18b6191614
COVID-824.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-824.png,COVID/images/COVID-824.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-824.png,COVID/masks/COVID-824.png,True,False,299,299,L,94.05,4e8ada71d836a9b0205d41eff7b4c4bf
COVID-825.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-825.png,COVID/images/COVID-825.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-825.png,COVID/masks/COVID-825.png,True,False,299,299,L,147.73,3f80a4d13e0bdc4c5e967c4f697ce041
COVID-826.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-826.png,COVID/images/COVID-826.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-826.png,COVID/masks/COVID-826.png,True,False,299,299,L,150.42,9f8ead02227924aa64308b983f7100f5
COVID-827.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-827.png,COVID/images/COVID-827.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-827.png,COVID/masks/COVID-827.png,True,False,299,299,L,128.87,3b853ddc8b74f35f15930f0d0cf8fbd0
COVID-828.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-828.png,COVID/images/COVID-828.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-828.png,COVID/masks/COVID-828.png,True,False,299,299,L,151.57,5628675d55df942df0016d21f32e324d
COVID-829.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-829.png,COVID/images/COVID-829.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-829.png,COVID/masks/COVID-829.png,True,False,299,299,L,151.42,e2a323b532178af0611b5d6d343744ba
COVID-83.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-83.png,COVID/images/COVID-83.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-83.png,COVID/masks/COVID-83.png,True,False,299,299,L,143.23,429fe0f9c7d0c9d7b287707fcad3941d
COVID-830.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-830.png,COVID/images/COVID-830.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-830.png,COVID/masks/COVID-830.png,True,False,299,299,L,118.55,43a37810b5fe9b3304f4fa5b502602d3
COVID-831.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-831.png,COVID/images/COVID-831.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-831.png,COVID/masks/COVID-831.png,True,False,299,299,L,151.35,ba896e4d4a6404da5b74fe925346e815
COVID-832.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-832.png,COVID/images/COVID-832.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-832.png,COVID/masks/COVID-832.png,True,False,299,299,L,125.98,9128b54fba373e88cb6678f771be953e
COVID-833.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-833.png,COVID/images/COVID-833.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-833.png,COVID/masks/COVID-833.png,True,False,299,299,L,186.35,5f047ea1cc31c71e327d34da67d5cab5
COVID-834.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-834.png,COVID/images/COVID-834.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-834.png,COVID/masks/COVID-834.png,True,False,299,299,L,146.41,33518344f3f6aa71f17bf5d476721276
COVID-835.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-835.png,COVID/images/COVID-835.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-835.png,COVID/masks/COVID-835.png,True,False,299,299,L,141.27,124b3d1bb2eefb2ea643c1eade985a3e
COVID-836.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-836.png,COVID/images/COVID-836.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-836.png,COVID/masks/COVID-836.png,True,False,299,299,L,159.49,0266c70737db05a8c725f5b95581266d
COVID-837.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-837.png,COVID/images/COVID-837.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-837.png,COVID/masks/COVID-837.png,True,False,299,299,L,133.41,d1badb7dcec34bfc3f092c51a2a7d72b
COVID-838.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-838.png,COVID/images/COVID-838.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-838.png,COVID/masks/COVID-838.png,True,False,299,299,L,144.05,99a08f38ffbe3883f28b9dfa5b9276da
COVID-839.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-839.png,COVID/images/COVID-839.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-839.png,COVID/masks/COVID-839.png,True,False,299,299,L,121.24,2f38650dbefea09525843d7165770b73
COVID-84.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-84.png,COVID/images/COVID-84.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-84.png,COVID/masks/COVID-84.png,True,False,299,299,L,153.75,86492f84a872b7d8d28f868c5f4bbbcf
COVID-840.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-840.png,COVID/images/COVID-840.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-840.png,COVID/masks/COVID-840.png,True,False,299,299,L,133.91,a497ceaeed065c93ed3b62568bdf3c84
COVID-841.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-841.png,COVID/images/COVID-841.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-841.png,COVID/masks/COVID-841.png,True,False,299,299,L,129.78,b58916f1bf0813fdf1f33eff76de1db3
COVID-842.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-842.png,COVID/images/COVID-842.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-842.png,COVID/masks/COVID-842.png,True,False,299,299,L,127.72,91feb136e43c4b5c411683968cd7e67a
COVID-843.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-843.png,COVID/images/COVID-843.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-843.png,COVID/masks/COVID-843.png,True,False,299,299,L,176.63,ce57bec8aee32abb7faaa0e60164ffe7
COVID-844.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-844.png,COVID/images/COVID-844.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-844.png,COVID/masks/COVID-844.png,True,False,299,299,L,129.67,481013b6e3809cf182a867d3450b7f77
COVID-845.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-845.png,COVID/images/COVID-845.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-845.png,COVID/masks/COVID-845.png,True,False,299,299,L,143.32,8d6f0df8cec9e0f537ec07ba759a0578
COVID-846.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-846.png,COVID/images/COVID-846.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-846.png,COVID/masks/COVID-846.png,True,False,299,299,L,151.8,882b7941f347ccabdf1721cc28679198
COVID-847.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-847.png,COVID/images/COVID-847.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-847.png,COVID/masks/COVID-847.png,True,False,299,299,L,152.05,6b5ad7caa73bd30fd34e62e62a8e9444
COVID-848.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-848.png,COVID/images/COVID-848.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-848.png,COVID/masks/COVID-848.png,True,False,299,299,L,152.76,3271b8363abde82d1945b543b35dca4e
COVID-849.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-849.png,COVID/images/COVID-849.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-849.png,COVID/masks/COVID-849.png,True,False,299,299,L,158.34,626b3c7edee9177a07937e901c5d0bec
COVID-85.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-85.png,COVID/images/COVID-85.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-85.png,COVID/masks/COVID-85.png,True,False,299,299,L,169.29,21c7f4aaca515ba338aa42a8d96d908c
COVID-850.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-850.png,COVID/images/COVID-850.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-850.png,COVID/masks/COVID-850.png,True,False,299,299,L,126.57,709edf62f39aacc3718f0bbc9cb883ca
COVID-851.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-851.png,COVID/images/COVID-851.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-851.png,COVID/masks/COVID-851.png,True,False,299,299,L,144.37,cd4dc0bcd60099e74145698e7584cfbe
COVID-852.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-852.png,COVID/images/COVID-852.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-852.png,COVID/masks/COVID-852.png,True,False,299,299,L,131.56,249e3e3399cba2340237abede5666aee
COVID-853.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-853.png,COVID/images/COVID-853.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-853.png,COVID/masks/COVID-853.png,True,False,299,299,L,141.85,69f6d9aa0cbe5a8854f179dde3ad7e2f
COVID-854.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-854.png,COVID/images/COVID-854.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-854.png,COVID/masks/COVID-854.png,True,False,299,299,L,150.04,c2ea5fddf09475d3ee0dd8ecbe8746b9
COVID-855.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-855.png,COVID/images/COVID-855.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-855.png,COVID/masks/COVID-855.png,True,False,299,299,L,136.26,f76603e4682d6e60a64331b45b52cdf4
COVID-856.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-856.png,COVID/images/COVID-856.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-856.png,COVID/masks/COVID-856.png,True,False,299,299,L,148.71,dbfd32804d5d6ec5ebd13a02280e5fbe
COVID-857.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-857.png,COVID/images/COVID-857.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-857.png,COVID/masks/COVID-857.png,True,False,299,299,L,140.83,29d292c2fa2d91330af04ed8ee92d893
COVID-858.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-858.png,COVID/images/COVID-858.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-858.png,COVID/masks/COVID-858.png,True,False,299,299,L,141.38,02dfd8b89c3e3d8a47c0956555497027
COVID-859.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-859.png,COVID/images/COVID-859.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-859.png,COVID/masks/COVID-859.png,True,False,299,299,L,127.61,afc7296f236a3b6ad69bc17366e1b1e7
COVID-86.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-86.png,COVID/images/COVID-86.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-86.png,COVID/masks/COVID-86.png,True,False,299,299,L,101.42,4e5903401ae06bfc7436fb773e1b8309
COVID-860.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-860.png,COVID/images/COVID-860.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-860.png,COVID/masks/COVID-860.png,True,False,299,299,L,157.75,f2e4ae414f6c80b0be3a1e5c1e774000
COVID-861.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-861.png,COVID/images/COVID-861.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-861.png,COVID/masks/COVID-861.png,True,False,299,299,L,134.65,dcdc2c31aa72773995c7cbd7ef25ec34
COVID-862.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-862.png,COVID/images/COVID-862.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-862.png,COVID/masks/COVID-862.png,True,False,299,299,L,150.59,f62fa7ab64bf3a5a3816766562b5263d
COVID-863.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-863.png,COVID/images/COVID-863.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-863.png,COVID/masks/COVID-863.png,True,False,299,299,L,128.11,2c3ca78cf0c220cf0dc02d8e981bd699
COVID-864.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-864.png,COVID/images/COVID-864.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-864.png,COVID/masks/COVID-864.png,True,False,299,299,L,124.78,272124a87ccd9880240d475894fe65bf
COVID-865.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-865.png,COVID/images/COVID-865.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-865.png,COVID/masks/COVID-865.png,True,False,299,299,L,163.09,e3cb2f8d8784911f64343e4be2157d98
COVID-866.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-866.png,COVID/images/COVID-866.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-866.png,COVID/masks/COVID-866.png,True,False,299,299,L,127.46,fbe983eba707843734bbab0c09c0471b
COVID-867.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-867.png,COVID/images/COVID-867.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-867.png,COVID/masks/COVID-867.png,True,False,299,299,L,157.43,e06ee00352da6aad9e3b5ee1e277ed8b
COVID-868.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-868.png,COVID/images/COVID-868.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-868.png,COVID/masks/COVID-868.png,True,False,299,299,L,145.23,21de73578fbe01aefa965877a569ae22
COVID-869.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-869.png,COVID/images/COVID-869.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-869.png,COVID/masks/COVID-869.png,True,False,299,299,L,143.98,00370d6ce9b836de5f20e931802640a6
COVID-87.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-87.png,COVID/images/COVID-87.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-87.png,COVID/masks/COVID-87.png,True,False,299,299,L,131.86,8761d1e481cf10a7cbf731676f380b78
COVID-870.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-870.png,COVID/images/COVID-870.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-870.png,COVID/masks/COVID-870.png,True,False,299,299,L,167.66,c24a4f43d0f6c1d0087d35ac8f4782fc
COVID-871.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-871.png,COVID/images/COVID-871.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-871.png,COVID/masks/COVID-871.png,True,False,299,299,L,138.87,41b000429d0f8232e8ba1f2f48637523
COVID-872.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-872.png,COVID/images/COVID-872.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-872.png,COVID/masks/COVID-872.png,True,False,299,299,L,126.1,e3a77cb318343c7d86a1ce5d03b8d174
COVID-873.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-873.png,COVID/images/COVID-873.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-873.png,COVID/masks/COVID-873.png,True,False,299,299,L,158.33,b885ab6937cd9d6a59e7bf3ba2fe5bc3
COVID-874.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-874.png,COVID/images/COVID-874.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-874.png,COVID/masks/COVID-874.png,True,False,299,299,L,158.74,6dc27255ff977489cd8b485fe1bde700
COVID-875.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-875.png,COVID/images/COVID-875.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-875.png,COVID/masks/COVID-875.png,True,False,299,299,L,147.86,d7ed06641e9746d4732f61670e0a07e8
COVID-876.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-876.png,COVID/images/COVID-876.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-876.png,COVID/masks/COVID-876.png,True,False,299,299,L,128.62,01fb32ff4dfbc443cdfc5842cb95a658
COVID-877.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-877.png,COVID/images/COVID-877.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-877.png,COVID/masks/COVID-877.png,True,False,299,299,L,151.52,e4d494567f93d0e7627dd09c0c8cda13
COVID-878.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-878.png,COVID/images/COVID-878.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-878.png,COVID/masks/COVID-878.png,True,False,299,299,L,162.59,acc5510d7c5c404ef28d965a7e38f66a
COVID-879.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-879.png,COVID/images/COVID-879.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-879.png,COVID/masks/COVID-879.png,True,False,299,299,L,148.51,e5171072e6286ac73c5d54e9c226c313
COVID-88.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-88.png,COVID/images/COVID-88.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-88.png,COVID/masks/COVID-88.png,True,False,299,299,L,170.34,d2a0247e2981c1dae0c70c782740deb9
COVID-880.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-880.png,COVID/images/COVID-880.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-880.png,COVID/masks/COVID-880.png,True,False,299,299,L,190.6,f7211b9e6740ce142068c41fac473c40
COVID-881.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-881.png,COVID/images/COVID-881.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-881.png,COVID/masks/COVID-881.png,True,False,299,299,L,137.88,7e8e62ff4697bfa64813501a80c36355
COVID-882.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-882.png,COVID/images/COVID-882.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-882.png,COVID/masks/COVID-882.png,True,False,299,299,L,160.45,2330696f70323991ca9711be3405ce08
COVID-883.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-883.png,COVID/images/COVID-883.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-883.png,COVID/masks/COVID-883.png,True,False,299,299,L,145.7,f9d09253e819afa86fdb607c9f3f57ab
COVID-884.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-884.png,COVID/images/COVID-884.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-884.png,COVID/masks/COVID-884.png,True,False,299,299,L,149.17,cbd72d3aa2ad26a40508950d71009f8b
COVID-885.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-885.png,COVID/images/COVID-885.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-885.png,COVID/masks/COVID-885.png,True,False,299,299,L,176.14,c54060e176174fb65837e68f955aa708
COVID-886.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-886.png,COVID/images/COVID-886.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-886.png,COVID/masks/COVID-886.png,True,False,299,299,L,121.38,5285515cf7cc076a989e54159c3bd2e7
COVID-887.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-887.png,COVID/images/COVID-887.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-887.png,COVID/masks/COVID-887.png,True,False,299,299,L,149.66,ac5de1d4777d04a0252cf2dec040b448
COVID-888.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-888.png,COVID/images/COVID-888.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-888.png,COVID/masks/COVID-888.png,True,False,299,299,L,142.19,83ffbb003bb4f682c9f9b659e5c28968
COVID-889.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-889.png,COVID/images/COVID-889.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-889.png,COVID/masks/COVID-889.png,True,False,299,299,L,112.9,39cd9ff0b92063bd0830529e8f0188a9
COVID-89.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-89.png,COVID/images/COVID-89.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-89.png,COVID/masks/COVID-89.png,True,False,299,299,L,164.23,af0f7c81fe7dcea67ed705d2fc6af1d9
COVID-890.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-890.png,COVID/images/COVID-890.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-890.png,COVID/masks/COVID-890.png,True,False,299,299,L,147.27,33625c83609e9808a09320182affc170
COVID-891.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-891.png,COVID/images/COVID-891.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-891.png,COVID/masks/COVID-891.png,True,False,299,299,L,138.52,9e038cb6a1bec7e53840e9858846dde3
COVID-892.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-892.png,COVID/images/COVID-892.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-892.png,COVID/masks/COVID-892.png,True,False,299,299,L,151.35,afb245b48284179a4d6ee4c7e7590c91
COVID-893.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-893.png,COVID/images/COVID-893.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-893.png,COVID/masks/COVID-893.png,True,False,299,299,L,149.7,01d4ee83c4e37ef0a398356df691cca8
COVID-894.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-894.png,COVID/images/COVID-894.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-894.png,COVID/masks/COVID-894.png,True,False,299,299,L,145.28,c5c91fa53177cd5c653a010a77ff47a7
COVID-895.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-895.png,COVID/images/COVID-895.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-895.png,COVID/masks/COVID-895.png,True,False,299,299,L,125.23,25076edcf07005b5b066a5d6acba7b6b
COVID-896.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-896.png,COVID/images/COVID-896.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-896.png,COVID/masks/COVID-896.png,True,False,299,299,L,146.92,a664a0ff16202cc41d006752c08ddf43
COVID-897.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-897.png,COVID/images/COVID-897.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-897.png,COVID/masks/COVID-897.png,True,False,299,299,L,154.69,e3a864a8532033100e735863d8178bee
COVID-898.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-898.png,COVID/images/COVID-898.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-898.png,COVID/masks/COVID-898.png,True,False,299,299,L,126.23,6f1c345b61903fa3582632fc87f97365
COVID-899.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-899.png,COVID/images/COVID-899.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-899.png,COVID/masks/COVID-899.png,True,False,299,299,L,160.46,1cc46615d8ec61fd68f5c1dcdc5d7615
COVID-9.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-9.png,COVID/images/COVID-9.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-9.png,COVID/masks/COVID-9.png,True,False,299,299,L,125.14,70009596389669d54262dc59f5f471ce
COVID-90.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-90.png,COVID/images/COVID-90.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-90.png,COVID/masks/COVID-90.png,True,False,299,299,L,171.91,52950c97b67eba7754867189523d82ae
COVID-900.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-900.png,COVID/images/COVID-900.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-900.png,COVID/masks/COVID-900.png,True,False,299,299,L,170.58,50d64cbecef0606b90de70186151dba1
COVID-901.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-901.png,COVID/images/COVID-901.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-901.png,COVID/masks/COVID-901.png,True,False,299,299,L,126.83,f37d6eb0b4be51afb4c4160715f10a2b
COVID-902.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-902.png,COVID/images/COVID-902.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-902.png,COVID/masks/COVID-902.png,True,False,299,299,L,162.39,65e9d9f3a510b9227a3be1b0791a492a
COVID-903.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-903.png,COVID/images/COVID-903.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-903.png,COVID/masks/COVID-903.png,True,False,299,299,L,205.91,5d24dc2bd837281a24ada31bd31e620a
COVID-904.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-904.png,COVID/images/COVID-904.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-904.png,COVID/masks/COVID-904.png,True,False,299,299,L,133.96,6dbfc8441716aec6f21d5de71b67e82a
COVID-905.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-905.png,COVID/images/COVID-905.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-905.png,COVID/masks/COVID-905.png,True,False,299,299,L,116.68,0f67d83e9da6a9944ee9aaf63b9918b8
COVID-906.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-906.png,COVID/images/COVID-906.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-906.png,COVID/masks/COVID-906.png,True,False,299,299,L,142.18,90b06c6e3d564681d0d08187d6abd60a
COVID-907.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-907.png,COVID/images/COVID-907.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-907.png,COVID/masks/COVID-907.png,True,False,299,299,L,141.79,26cc39c5dfd3032fc88c9c6c54f3e0f2
COVID-908.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-908.png,COVID/images/COVID-908.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-908.png,COVID/masks/COVID-908.png,True,False,299,299,L,170.51,a3c894c538e7f8067d102a7595919d7a
COVID-909.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-909.png,COVID/images/COVID-909.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-909.png,COVID/masks/COVID-909.png,True,False,299,299,L,150.2,f352c1f89b3aa1ce0ec11291086009aa
COVID-91.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-91.png,COVID/images/COVID-91.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-91.png,COVID/masks/COVID-91.png,True,False,299,299,L,156.1,e6ab66ae18c915b52df0b43ec16c86d9
COVID-910.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-910.png,COVID/images/COVID-910.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-910.png,COVID/masks/COVID-910.png,True,False,299,299,L,173.35,64b54bfc3812a74bb9e174a5cfb6075e
COVID-911.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-911.png,COVID/images/COVID-911.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-911.png,COVID/masks/COVID-911.png,True,False,299,299,L,143.62,e853d85e3efcb0648f335bd0fe42f777
COVID-912.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-912.png,COVID/images/COVID-912.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-912.png,COVID/masks/COVID-912.png,True,False,299,299,L,154.41,9c1ded414c099bc81c52fb32021d7497
COVID-913.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-913.png,COVID/images/COVID-913.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-913.png,COVID/masks/COVID-913.png,True,False,299,299,L,142.9,e20635b03dadc882171b341a9d08e3f9
COVID-914.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-914.png,COVID/images/COVID-914.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-914.png,COVID/masks/COVID-914.png,True,False,299,299,L,127.08,1a288a0dff0f2e7821aa6253c3518950
COVID-915.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-915.png,COVID/images/COVID-915.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-915.png,COVID/masks/COVID-915.png,True,False,299,299,L,169.96,f879d35004ad117d3f4cc5e6df3069a7
COVID-916.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-916.png,COVID/images/COVID-916.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-916.png,COVID/masks/COVID-916.png,True,False,299,299,L,122.85,68be317d768d0fe978bd1b5c43c3b967
COVID-917.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-917.png,COVID/images/COVID-917.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-917.png,COVID/masks/COVID-917.png,True,False,299,299,L,165.03,6341e76ba1c5085aa661dcc96098146b
COVID-918.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-918.png,COVID/images/COVID-918.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-918.png,COVID/masks/COVID-918.png,True,False,299,299,L,135.47,33f1026bf5b9add23e04ef0b209d216e
COVID-919.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-919.png,COVID/images/COVID-919.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-919.png,COVID/masks/COVID-919.png,True,False,299,299,L,179.59,6b3cc79563562ef21debbfdeeceedd0d
COVID-92.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-92.png,COVID/images/COVID-92.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-92.png,COVID/masks/COVID-92.png,True,False,299,299,L,134.87,2262deee3794dda44e6095ac9e53912a
COVID-920.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-920.png,COVID/images/COVID-920.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-920.png,COVID/masks/COVID-920.png,True,False,299,299,L,133.46,b169912dfc4ba968bc43b1d5ada6a908
COVID-921.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-921.png,COVID/images/COVID-921.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-921.png,COVID/masks/COVID-921.png,True,False,299,299,L,155.42,66d776e786ebc77315346a2e166a3f7b
COVID-922.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-922.png,COVID/images/COVID-922.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-922.png,COVID/masks/COVID-922.png,True,False,299,299,L,97.9,a7ba6411011e88212f18cb7a90258e74
COVID-923.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-923.png,COVID/images/COVID-923.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-923.png,COVID/masks/COVID-923.png,True,False,299,299,L,134.54,728a2372b6c5eea501f22c6b6b75f411
COVID-924.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-924.png,COVID/images/COVID-924.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-924.png,COVID/masks/COVID-924.png,True,False,299,299,L,171.64,9c49ab391ab6ecc5c480bec48948e6bd
COVID-925.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-925.png,COVID/images/COVID-925.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-925.png,COVID/masks/COVID-925.png,True,False,299,299,L,111.65,2df56694b1851259eccb988dab3e9eb6
COVID-926.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-926.png,COVID/images/COVID-926.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-926.png,COVID/masks/COVID-926.png,True,False,299,299,L,156.12,a20d548f780240ce28b33285152c9505
COVID-927.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-927.png,COVID/images/COVID-927.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-927.png,COVID/masks/COVID-927.png,True,False,299,299,L,141.25,60b6eb9da5a7a38663d499213738e6a5
COVID-928.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-928.png,COVID/images/COVID-928.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-928.png,COVID/masks/COVID-928.png,True,False,299,299,L,124.75,6a89cd446af17f89f8f59766fd57b7e6
COVID-929.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-929.png,COVID/images/COVID-929.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-929.png,COVID/masks/COVID-929.png,True,False,299,299,L,136.44,c4bbcc88cf8a13e003d420fbbe55ba53
COVID-93.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-93.png,COVID/images/COVID-93.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-93.png,COVID/masks/COVID-93.png,True,False,299,299,L,150.73,269985393b3844ce7dbd630f0156fd4e
COVID-930.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-930.png,COVID/images/COVID-930.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-930.png,COVID/masks/COVID-930.png,True,False,299,299,L,140.56,286e022d7f5f9cd07c33ef6bcdaa90bb
COVID-931.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-931.png,COVID/images/COVID-931.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-931.png,COVID/masks/COVID-931.png,True,False,299,299,L,162.13,afc8ed502acb34f39fe9ae9c5590a9d7
COVID-932.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-932.png,COVID/images/COVID-932.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-932.png,COVID/masks/COVID-932.png,True,False,299,299,L,147.71,9a8115779b6acde40321215c18450fdd
COVID-933.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-933.png,COVID/images/COVID-933.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-933.png,COVID/masks/COVID-933.png,True,False,299,299,L,144.36,993e354f332ff2f8870fba4965ee5111
COVID-934.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-934.png,COVID/images/COVID-934.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-934.png,COVID/masks/COVID-934.png,True,False,299,299,L,149.95,9707ef8f81d71ca262d818bdfef649b2
COVID-935.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-935.png,COVID/images/COVID-935.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-935.png,COVID/masks/COVID-935.png,True,False,299,299,L,168.44,6ad57e38965db12509ddb625ee4bf927
COVID-936.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-936.png,COVID/images/COVID-936.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-936.png,COVID/masks/COVID-936.png,True,False,299,299,L,146.78,31f8d5ac99156a5aff98cae4a374cdb2
COVID-937.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-937.png,COVID/images/COVID-937.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-937.png,COVID/masks/COVID-937.png,True,False,299,299,L,143.76,4d5519d9d82254e0e3952f1af3147dda
COVID-938.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-938.png,COVID/images/COVID-938.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-938.png,COVID/masks/COVID-938.png,True,False,299,299,L,169.85,0cdf8cf9409a20c587c04335cf03138f
COVID-939.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-939.png,COVID/images/COVID-939.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-939.png,COVID/masks/COVID-939.png,True,False,299,299,L,127.47,fab52cec32a5682dc9e3e702c03ec3cf
COVID-94.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-94.png,COVID/images/COVID-94.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-94.png,COVID/masks/COVID-94.png,True,False,299,299,L,146.21,0f047e602051b716917eba651337e422
COVID-940.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-940.png,COVID/images/COVID-940.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-940.png,COVID/masks/COVID-940.png,True,False,299,299,L,130.52,1799ff8d378a535d7b52806fcb9d16e1
COVID-941.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-941.png,COVID/images/COVID-941.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-941.png,COVID/masks/COVID-941.png,True,False,299,299,L,137.46,448ece4261538021737709b9667e327b
COVID-942.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-942.png,COVID/images/COVID-942.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-942.png,COVID/masks/COVID-942.png,True,False,299,299,L,141.24,d8e9cd3e2dadbfe47ab5aefab8a35f90
COVID-943.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-943.png,COVID/images/COVID-943.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-943.png,COVID/masks/COVID-943.png,True,False,299,299,L,135.86,3dbb4f58a812765616633afc876c09c2
COVID-944.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-944.png,COVID/images/COVID-944.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-944.png,COVID/masks/COVID-944.png,True,False,299,299,L,189.28,d0444e7a630e728464f9e1686a67dd28
COVID-945.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-945.png,COVID/images/COVID-945.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-945.png,COVID/masks/COVID-945.png,True,False,299,299,L,126.0,1564e7143edf98ad9b855f7c3607e409
COVID-946.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-946.png,COVID/images/COVID-946.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-946.png,COVID/masks/COVID-946.png,True,False,299,299,L,163.6,4cf1e27e21eeb674210cfd6d3fa6b91d
COVID-947.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-947.png,COVID/images/COVID-947.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-947.png,COVID/masks/COVID-947.png,True,False,299,299,L,111.85,f1f8af8a4f8710109d57bf3961198be5
COVID-948.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-948.png,COVID/images/COVID-948.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-948.png,COVID/masks/COVID-948.png,True,False,299,299,L,133.26,09ff087d93b2b4c240e5ef0edde73865
COVID-949.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-949.png,COVID/images/COVID-949.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-949.png,COVID/masks/COVID-949.png,True,False,299,299,L,147.85,eae927cfdffd25bd77782d4e86246aa1
COVID-95.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-95.png,COVID/images/COVID-95.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-95.png,COVID/masks/COVID-95.png,True,False,299,299,L,134.88,08d5fe5992d89faa04951ce7857b43bb
COVID-950.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-950.png,COVID/images/COVID-950.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-950.png,COVID/masks/COVID-950.png,True,False,299,299,L,112.31,b467a10a216e8e71d98a3cc7b50a064c
COVID-951.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-951.png,COVID/images/COVID-951.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-951.png,COVID/masks/COVID-951.png,True,False,299,299,L,111.79,d0d744c6a4aece46d522d58c7fcdc3c0
COVID-952.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-952.png,COVID/images/COVID-952.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-952.png,COVID/masks/COVID-952.png,True,False,299,299,L,169.46,7ad65a80df454bf9d80ffe1bb689f8ce
COVID-953.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-953.png,COVID/images/COVID-953.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-953.png,COVID/masks/COVID-953.png,True,False,299,299,L,164.92,7c56067e50f0dc8574e81730a69e6b76
COVID-954.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-954.png,COVID/images/COVID-954.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-954.png,COVID/masks/COVID-954.png,True,False,299,299,L,147.04,8b0d92a323db17a2671d9c5b414e989c
COVID-955.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-955.png,COVID/images/COVID-955.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-955.png,COVID/masks/COVID-955.png,True,False,299,299,L,126.36,3e36f1f3190cbdc0fad8f22a3d0c6869
COVID-956.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-956.png,COVID/images/COVID-956.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-956.png,COVID/masks/COVID-956.png,True,False,299,299,L,164.02,9da27714ad177a029b1c805c7bf90f38
COVID-957.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-957.png,COVID/images/COVID-957.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-957.png,COVID/masks/COVID-957.png,True,False,299,299,L,153.06,8658d45a6bc786b5dfee293821ee5787
COVID-958.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-958.png,COVID/images/COVID-958.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-958.png,COVID/masks/COVID-958.png,True,False,299,299,L,131.69,ea2c43ea0da4d73f3ce57e290e0ddec6
COVID-959.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-959.png,COVID/images/COVID-959.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-959.png,COVID/masks/COVID-959.png,True,False,299,299,L,130.11,9b411c47161d8f258a2ed7805dd87051
COVID-96.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-96.png,COVID/images/COVID-96.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-96.png,COVID/masks/COVID-96.png,True,False,299,299,L,136.0,5bafa6868b3eab42d8266a3e025bb2ab
COVID-960.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-960.png,COVID/images/COVID-960.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-960.png,COVID/masks/COVID-960.png,True,False,299,299,L,139.28,bba074b9411ee4715153139a70290a54
COVID-961.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-961.png,COVID/images/COVID-961.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-961.png,COVID/masks/COVID-961.png,True,False,299,299,L,151.11,97c027fe6b6bfd3fb4d9f1323afd734d
COVID-962.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-962.png,COVID/images/COVID-962.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-962.png,COVID/masks/COVID-962.png,True,False,299,299,L,167.08,306ba2b63086341e6e4d7410595a016f
COVID-963.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-963.png,COVID/images/COVID-963.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-963.png,COVID/masks/COVID-963.png,True,False,299,299,L,148.96,10c20f881de7c375133ea6ac00bb6b14
COVID-964.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-964.png,COVID/images/COVID-964.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-964.png,COVID/masks/COVID-964.png,True,False,299,299,L,187.37,236c8e9da3d637630ae45d89e154a78b
COVID-965.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-965.png,COVID/images/COVID-965.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-965.png,COVID/masks/COVID-965.png,True,False,299,299,L,191.53,ef3fb6458da2a24a0731e61cba69998d
COVID-966.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-966.png,COVID/images/COVID-966.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-966.png,COVID/masks/COVID-966.png,True,False,299,299,L,141.0,32fa3083bd536f52dcf8e38362c08fc6
COVID-967.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-967.png,COVID/images/COVID-967.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-967.png,COVID/masks/COVID-967.png,True,False,299,299,L,152.68,cd623988ea97637d919e01e6606aa85d
COVID-968.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-968.png,COVID/images/COVID-968.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-968.png,COVID/masks/COVID-968.png,True,False,299,299,L,167.85,bc46bd2b1a5b76cd643590e4e40ab152
COVID-969.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-969.png,COVID/images/COVID-969.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-969.png,COVID/masks/COVID-969.png,True,False,299,299,L,144.62,515005a7f4b4d95ca85737ec6597ca21
COVID-97.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-97.png,COVID/images/COVID-97.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-97.png,COVID/masks/COVID-97.png,True,False,299,299,L,152.77,c6bdddf9cb6c43ea52fe73aa6b45664b
COVID-970.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-970.png,COVID/images/COVID-970.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-970.png,COVID/masks/COVID-970.png,True,False,299,299,L,157.79,7fbfb28dad702b5b0623a0dea8246fc9
COVID-971.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-971.png,COVID/images/COVID-971.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-971.png,COVID/masks/COVID-971.png,True,False,299,299,L,103.27,43201de572d501f73e544d2090956c90
COVID-972.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-972.png,COVID/images/COVID-972.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-972.png,COVID/masks/COVID-972.png,True,False,299,299,L,147.31,930a9e0963c4d4eb8c84af285ff84868
COVID-973.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-973.png,COVID/images/COVID-973.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-973.png,COVID/masks/COVID-973.png,True,False,299,299,L,148.58,8aff4faa8f7ff1c426499ff557041bc7
COVID-974.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-974.png,COVID/images/COVID-974.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-974.png,COVID/masks/COVID-974.png,True,False,299,299,L,162.14,97ecfbdd5feb79a955af75e4e37275a2
COVID-975.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-975.png,COVID/images/COVID-975.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-975.png,COVID/masks/COVID-975.png,True,False,299,299,L,131.45,1a33137e3046e246f391e27a5881351c
COVID-976.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-976.png,COVID/images/COVID-976.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-976.png,COVID/masks/COVID-976.png,True,False,299,299,L,129.79,e9974a5cd1e90da93b0fc72faa8a94cb
COVID-977.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-977.png,COVID/images/COVID-977.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-977.png,COVID/masks/COVID-977.png,True,False,299,299,L,123.84,321163790061b78fbcb14fb8d87066a2
COVID-978.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-978.png,COVID/images/COVID-978.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-978.png,COVID/masks/COVID-978.png,True,False,299,299,L,133.71,4d0c09f5444611ba97da9ebf34dd9dfa
COVID-979.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-979.png,COVID/images/COVID-979.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-979.png,COVID/masks/COVID-979.png,True,False,299,299,L,128.48,2fb54a975b4b158bb7fbf5da8fd5169f
COVID-98.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-98.png,COVID/images/COVID-98.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-98.png,COVID/masks/COVID-98.png,True,False,299,299,L,132.58,7cc16da40e14f6a095746e9c39ba85b3
COVID-980.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-980.png,COVID/images/COVID-980.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-980.png,COVID/masks/COVID-980.png,True,False,299,299,L,129.9,0195a5011dd3e3832adb050ad633d888
COVID-981.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-981.png,COVID/images/COVID-981.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-981.png,COVID/masks/COVID-981.png,True,False,299,299,L,102.74,30922175c44aa2143edf4699e5f5ccd2
COVID-982.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-982.png,COVID/images/COVID-982.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-982.png,COVID/masks/COVID-982.png,True,False,299,299,L,149.32,98ac3aa2ea3e53d249cf9d59f93737ee
COVID-983.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-983.png,COVID/images/COVID-983.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-983.png,COVID/masks/COVID-983.png,True,False,299,299,L,142.82,8af7c1911156c6c6dd25d52b14214caa
COVID-984.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-984.png,COVID/images/COVID-984.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-984.png,COVID/masks/COVID-984.png,True,False,299,299,L,148.76,890ae2cb2da2c833543ef5945c255a59
COVID-985.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-985.png,COVID/images/COVID-985.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-985.png,COVID/masks/COVID-985.png,True,False,299,299,L,122.3,f742aad430963bbd60ec48b701851125
COVID-986.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-986.png,COVID/images/COVID-986.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-986.png,COVID/masks/COVID-986.png,True,False,299,299,L,146.43,f5f2688c63fe3efef9b2c71607d23a72
COVID-987.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-987.png,COVID/images/COVID-987.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-987.png,COVID/masks/COVID-987.png,True,False,299,299,L,132.4,9a858585c1d99b6b3aa839c0f09a01aa
COVID-988.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-988.png,COVID/images/COVID-988.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-988.png,COVID/masks/COVID-988.png,True,False,299,299,L,104.73,4d3d7695dd161331a483b2d459130add
COVID-989.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-989.png,COVID/images/COVID-989.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-989.png,COVID/masks/COVID-989.png,True,False,299,299,L,133.26,bdd41dc2dc6b1fd217b517c835068f95
COVID-99.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-99.png,COVID/images/COVID-99.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-99.png,COVID/masks/COVID-99.png,True,False,299,299,L,136.44,1e3649e12b474829cfd7b37f903bdf60
COVID-990.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-990.png,COVID/images/COVID-990.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-990.png,COVID/masks/COVID-990.png,True,False,299,299,L,157.37,afef8b168b86058e156604356911bee8
COVID-991.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-991.png,COVID/images/COVID-991.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-991.png,COVID/masks/COVID-991.png,True,False,299,299,L,149.2,21e711eecf266df418f151b634b06b5e
COVID-992.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-992.png,COVID/images/COVID-992.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-992.png,COVID/masks/COVID-992.png,True,False,299,299,L,134.92,c84c341c25642e1c3540441520480369
COVID-993.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-993.png,COVID/images/COVID-993.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-993.png,COVID/masks/COVID-993.png,True,False,299,299,L,153.8,665cdec947e2cc0a2f2b62b6f2bfc3f7
COVID-994.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-994.png,COVID/images/COVID-994.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-994.png,COVID/masks/COVID-994.png,True,False,299,299,L,150.01,137e0d08854accc1638159fd57ea95a7
COVID-995.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-995.png,COVID/images/COVID-995.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-995.png,COVID/masks/COVID-995.png,True,False,299,299,L,146.75,21b0c8c4f954e91a0518963346e45131
COVID-996.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-996.png,COVID/images/COVID-996.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-996.png,COVID/masks/COVID-996.png,True,False,299,299,L,141.03,8c01064cbf44903923b25a63018f9aff
COVID-997.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-997.png,COVID/images/COVID-997.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-997.png,COVID/masks/COVID-997.png,True,False,299,299,L,119.46,1aa81932fdd81441d288368610d540bc
COVID-998.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-998.png,COVID/images/COVID-998.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-998.png,COVID/masks/COVID-998.png,True,False,299,299,L,134.76,25e8ac5547dd469454d424987a124fe6
COVID-999.png,COVID,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images/COVID-999.png,COVID/images/COVID-999.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/masks/COVID-999.png,COVID/masks/COVID-999.png,True,False,299,299,L,160.84,6989f277b0772f6069b8082bc834b8d5
Lung_Opacity-1.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1.png,Lung_Opacity/images/Lung_Opacity-1.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1.png,Lung_Opacity/masks/Lung_Opacity-1.png,True,False,299,299,L,94.55,10b9264097281fe68d48259eb5e9cfd7
Lung_Opacity-10.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-10.png,Lung_Opacity/images/Lung_Opacity-10.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-10.png,Lung_Opacity/masks/Lung_Opacity-10.png,True,False,299,299,L,162.39,f626b5dc6f37bd8fd1bd80ca94b08c85
Lung_Opacity-100.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-100.png,Lung_Opacity/images/Lung_Opacity-100.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-100.png,Lung_Opacity/masks/Lung_Opacity-100.png,True,False,299,299,L,119.09,6d0e39230cc9ff46cbfe012d056ff505
Lung_Opacity-1000.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1000.png,Lung_Opacity/images/Lung_Opacity-1000.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1000.png,Lung_Opacity/masks/Lung_Opacity-1000.png,True,False,299,299,L,150.94,32d94800ecbc8b302a0fc3ceace2d24b
Lung_Opacity-1001.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1001.png,Lung_Opacity/images/Lung_Opacity-1001.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1001.png,Lung_Opacity/masks/Lung_Opacity-1001.png,True,False,299,299,L,110.66,35cc90abf7d3b748b67769f2f3b198e0
Lung_Opacity-1002.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1002.png,Lung_Opacity/images/Lung_Opacity-1002.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1002.png,Lung_Opacity/masks/Lung_Opacity-1002.png,True,False,299,299,L,110.66,9b5fcf2103ea80eeb965de08d7ee4eed
Lung_Opacity-1003.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1003.png,Lung_Opacity/images/Lung_Opacity-1003.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1003.png,Lung_Opacity/masks/Lung_Opacity-1003.png,True,False,299,299,L,131.0,56d110747a17b46c5cb97ae0cb095c7a
Lung_Opacity-1004.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1004.png,Lung_Opacity/images/Lung_Opacity-1004.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1004.png,Lung_Opacity/masks/Lung_Opacity-1004.png,True,False,299,299,L,116.56,d07407783557ed1fe9a3f58f7446a713
Lung_Opacity-1005.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1005.png,Lung_Opacity/images/Lung_Opacity-1005.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1005.png,Lung_Opacity/masks/Lung_Opacity-1005.png,True,False,299,299,L,115.0,eb505071a48fe0903b69c18eccae9212
Lung_Opacity-1006.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1006.png,Lung_Opacity/images/Lung_Opacity-1006.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1006.png,Lung_Opacity/masks/Lung_Opacity-1006.png,True,False,299,299,L,128.24,39cf8b352d50a41cd28abedfb5275384
Lung_Opacity-1007.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1007.png,Lung_Opacity/images/Lung_Opacity-1007.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1007.png,Lung_Opacity/masks/Lung_Opacity-1007.png,True,False,299,299,L,130.79,31ba809ce545a7941f4355034066470b
Lung_Opacity-1008.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1008.png,Lung_Opacity/images/Lung_Opacity-1008.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1008.png,Lung_Opacity/masks/Lung_Opacity-1008.png,True,False,299,299,L,122.28,1e0520629a5ea41a424f31a59a05fee2
Lung_Opacity-1009.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1009.png,Lung_Opacity/images/Lung_Opacity-1009.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1009.png,Lung_Opacity/masks/Lung_Opacity-1009.png,True,False,299,299,L,116.36,de7ad373ead9f9c6607c514ecec5dd3d
Lung_Opacity-101.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-101.png,Lung_Opacity/images/Lung_Opacity-101.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-101.png,Lung_Opacity/masks/Lung_Opacity-101.png,True,False,299,299,L,118.79,a96117e8501841740cf80c63fde3c9f2
Lung_Opacity-1010.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1010.png,Lung_Opacity/images/Lung_Opacity-1010.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1010.png,Lung_Opacity/masks/Lung_Opacity-1010.png,True,False,299,299,L,129.99,9b160743bb519681a0ac62159ea6d18d
Lung_Opacity-1011.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1011.png,Lung_Opacity/images/Lung_Opacity-1011.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1011.png,Lung_Opacity/masks/Lung_Opacity-1011.png,True,False,299,299,L,105.88,f1b35820dcf4d95584b3a03687f6d7fe
Lung_Opacity-1012.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1012.png,Lung_Opacity/images/Lung_Opacity-1012.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1012.png,Lung_Opacity/masks/Lung_Opacity-1012.png,True,False,299,299,L,130.53,8f49ce78c79ee828348bf36cc46c1369
Lung_Opacity-1013.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1013.png,Lung_Opacity/images/Lung_Opacity-1013.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1013.png,Lung_Opacity/masks/Lung_Opacity-1013.png,True,False,299,299,L,171.76,133b7cfbbf037778542cc6fd2a4ff16a
Lung_Opacity-1014.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1014.png,Lung_Opacity/images/Lung_Opacity-1014.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1014.png,Lung_Opacity/masks/Lung_Opacity-1014.png,True,False,299,299,L,112.93,f0a2632c2e404e98982d7cba27326372
Lung_Opacity-1015.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1015.png,Lung_Opacity/images/Lung_Opacity-1015.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1015.png,Lung_Opacity/masks/Lung_Opacity-1015.png,True,False,299,299,L,87.23,b0c788eba1cf7c2ef8e8fb20481e3efd
Lung_Opacity-1016.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1016.png,Lung_Opacity/images/Lung_Opacity-1016.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1016.png,Lung_Opacity/masks/Lung_Opacity-1016.png,True,False,299,299,L,93.35,16fdd0c5f3e3dfb0cc53b6a6fb5f93c6
Lung_Opacity-1017.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1017.png,Lung_Opacity/images/Lung_Opacity-1017.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1017.png,Lung_Opacity/masks/Lung_Opacity-1017.png,True,False,299,299,L,130.64,ad43626381d22e52b31463de075fd3d9
Lung_Opacity-1018.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1018.png,Lung_Opacity/images/Lung_Opacity-1018.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1018.png,Lung_Opacity/masks/Lung_Opacity-1018.png,True,False,299,299,L,96.42,624b81c8b34f0079a454f18b24d63840
Lung_Opacity-1019.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1019.png,Lung_Opacity/images/Lung_Opacity-1019.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1019.png,Lung_Opacity/masks/Lung_Opacity-1019.png,True,False,299,299,L,155.11,bbbb5af0a1ddcfec5e490db58d915333
Lung_Opacity-102.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-102.png,Lung_Opacity/images/Lung_Opacity-102.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-102.png,Lung_Opacity/masks/Lung_Opacity-102.png,True,False,299,299,L,126.96,5de24ceaec8181624ca9550622ee7717
Lung_Opacity-1020.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1020.png,Lung_Opacity/images/Lung_Opacity-1020.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1020.png,Lung_Opacity/masks/Lung_Opacity-1020.png,True,False,299,299,L,122.72,b542b77ce319ec369a277088862b103c
Lung_Opacity-1021.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1021.png,Lung_Opacity/images/Lung_Opacity-1021.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1021.png,Lung_Opacity/masks/Lung_Opacity-1021.png,True,False,299,299,L,103.28,ce7712a0c15e552dd2cf5c7a61720137
Lung_Opacity-1022.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1022.png,Lung_Opacity/images/Lung_Opacity-1022.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1022.png,Lung_Opacity/masks/Lung_Opacity-1022.png,True,False,299,299,L,168.31,bea7cd1db880c3815d41f4612a4841fe
Lung_Opacity-1023.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1023.png,Lung_Opacity/images/Lung_Opacity-1023.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1023.png,Lung_Opacity/masks/Lung_Opacity-1023.png,True,False,299,299,L,113.23,979c45aef69ee01be38886e51ed81c0e
Lung_Opacity-1024.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1024.png,Lung_Opacity/images/Lung_Opacity-1024.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1024.png,Lung_Opacity/masks/Lung_Opacity-1024.png,True,False,299,299,L,118.26,46726300ed7c85fae438ca7d72deba60
Lung_Opacity-1025.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1025.png,Lung_Opacity/images/Lung_Opacity-1025.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1025.png,Lung_Opacity/masks/Lung_Opacity-1025.png,True,False,299,299,L,151.67,58852638a9ab06861072bddedc13f081
Lung_Opacity-1026.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1026.png,Lung_Opacity/images/Lung_Opacity-1026.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1026.png,Lung_Opacity/masks/Lung_Opacity-1026.png,True,False,299,299,L,178.34,abb4d9ecb0822adfea424046dd9b8948
Lung_Opacity-1027.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1027.png,Lung_Opacity/images/Lung_Opacity-1027.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1027.png,Lung_Opacity/masks/Lung_Opacity-1027.png,True,False,299,299,L,122.49,219a6a5c0c6f23f9d49a2130851aaa56
Lung_Opacity-1028.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1028.png,Lung_Opacity/images/Lung_Opacity-1028.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1028.png,Lung_Opacity/masks/Lung_Opacity-1028.png,True,False,299,299,L,102.95,e2827ed5e674aa063f12cc19cb423179
Lung_Opacity-1029.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1029.png,Lung_Opacity/images/Lung_Opacity-1029.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1029.png,Lung_Opacity/masks/Lung_Opacity-1029.png,True,False,299,299,L,102.21,a7c8bf16f334fd907d2a5f40a6c7c1a3
Lung_Opacity-103.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-103.png,Lung_Opacity/images/Lung_Opacity-103.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-103.png,Lung_Opacity/masks/Lung_Opacity-103.png,True,False,299,299,L,123.16,7f5cc963647b75746400b4e857ffa488
Lung_Opacity-1030.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1030.png,Lung_Opacity/images/Lung_Opacity-1030.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1030.png,Lung_Opacity/masks/Lung_Opacity-1030.png,True,False,299,299,L,148.99,e5f7512688126dc1c692150991a207d6
Lung_Opacity-1031.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1031.png,Lung_Opacity/images/Lung_Opacity-1031.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1031.png,Lung_Opacity/masks/Lung_Opacity-1031.png,True,False,299,299,L,105.7,21420e3f86412927390a1b4b368a74e3
Lung_Opacity-1032.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1032.png,Lung_Opacity/images/Lung_Opacity-1032.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1032.png,Lung_Opacity/masks/Lung_Opacity-1032.png,True,False,299,299,L,112.98,c573e08bf7d23c0a091b3d69858943b4
Lung_Opacity-1033.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1033.png,Lung_Opacity/images/Lung_Opacity-1033.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1033.png,Lung_Opacity/masks/Lung_Opacity-1033.png,True,False,299,299,L,144.33,d60c0b4774e1d2dfd039bc451594185a
Lung_Opacity-1034.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1034.png,Lung_Opacity/images/Lung_Opacity-1034.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1034.png,Lung_Opacity/masks/Lung_Opacity-1034.png,True,False,299,299,L,108.59,064f4c6337e24329481ff59706058f56
Lung_Opacity-1035.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1035.png,Lung_Opacity/images/Lung_Opacity-1035.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1035.png,Lung_Opacity/masks/Lung_Opacity-1035.png,True,False,299,299,L,169.97,672ae00ed903c5435bb2cad99db09a85
Lung_Opacity-1036.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1036.png,Lung_Opacity/images/Lung_Opacity-1036.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1036.png,Lung_Opacity/masks/Lung_Opacity-1036.png,True,False,299,299,L,106.3,6759eef2ec48f59d0adaa789ca46cd08
Lung_Opacity-1037.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1037.png,Lung_Opacity/images/Lung_Opacity-1037.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1037.png,Lung_Opacity/masks/Lung_Opacity-1037.png,True,False,299,299,L,160.66,d3a2013a322c1e7fdcd561a3325f9fb7
Lung_Opacity-1038.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1038.png,Lung_Opacity/images/Lung_Opacity-1038.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1038.png,Lung_Opacity/masks/Lung_Opacity-1038.png,True,False,299,299,L,113.89,7560c3a2099e61d80af2b89beb7da37b
Lung_Opacity-1039.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1039.png,Lung_Opacity/images/Lung_Opacity-1039.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1039.png,Lung_Opacity/masks/Lung_Opacity-1039.png,True,False,299,299,L,112.06,37b01cde4c5f36e5d27fdd17b7397e85
Lung_Opacity-104.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-104.png,Lung_Opacity/images/Lung_Opacity-104.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-104.png,Lung_Opacity/masks/Lung_Opacity-104.png,True,False,299,299,L,110.4,b52b6934e3de71e8ce3ea66daf687f3a
Lung_Opacity-1040.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1040.png,Lung_Opacity/images/Lung_Opacity-1040.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1040.png,Lung_Opacity/masks/Lung_Opacity-1040.png,True,False,299,299,L,112.36,8b6de64bdb6ca2d361812408300e91b6
Lung_Opacity-1041.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1041.png,Lung_Opacity/images/Lung_Opacity-1041.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1041.png,Lung_Opacity/masks/Lung_Opacity-1041.png,True,False,299,299,L,114.21,cd2993f6ea934023da6052d5884f6f68
Lung_Opacity-1042.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1042.png,Lung_Opacity/images/Lung_Opacity-1042.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1042.png,Lung_Opacity/masks/Lung_Opacity-1042.png,True,False,299,299,L,111.98,fb627f784e63f13d9e5c3cf094c269ca
Lung_Opacity-1043.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1043.png,Lung_Opacity/images/Lung_Opacity-1043.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1043.png,Lung_Opacity/masks/Lung_Opacity-1043.png,True,False,299,299,L,109.91,4e897b3cfffa6754a9bd663c41f1b946
Lung_Opacity-1044.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1044.png,Lung_Opacity/images/Lung_Opacity-1044.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1044.png,Lung_Opacity/masks/Lung_Opacity-1044.png,True,False,299,299,L,102.77,2e480bc9e674b24c488a11a4bf0429dd
Lung_Opacity-1045.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1045.png,Lung_Opacity/images/Lung_Opacity-1045.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1045.png,Lung_Opacity/masks/Lung_Opacity-1045.png,True,False,299,299,L,106.19,bb112cc5cc138fb5053b163495fa49aa
Lung_Opacity-1046.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1046.png,Lung_Opacity/images/Lung_Opacity-1046.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1046.png,Lung_Opacity/masks/Lung_Opacity-1046.png,True,False,299,299,L,112.66,3b069d25feded8f1b9905d9ec332ba92
Lung_Opacity-1047.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1047.png,Lung_Opacity/images/Lung_Opacity-1047.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1047.png,Lung_Opacity/masks/Lung_Opacity-1047.png,True,False,299,299,L,119.2,6e2f8d9cfb3ef343141a0b980e0e2cfc
Lung_Opacity-1048.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1048.png,Lung_Opacity/images/Lung_Opacity-1048.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1048.png,Lung_Opacity/masks/Lung_Opacity-1048.png,True,False,299,299,L,169.25,b4ab6722d7e5ee9019039f5297c3550b
Lung_Opacity-1049.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1049.png,Lung_Opacity/images/Lung_Opacity-1049.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1049.png,Lung_Opacity/masks/Lung_Opacity-1049.png,True,False,299,299,L,119.93,450718b94d1ebee376a13ccedcccfb52
Lung_Opacity-105.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-105.png,Lung_Opacity/images/Lung_Opacity-105.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-105.png,Lung_Opacity/masks/Lung_Opacity-105.png,True,False,299,299,L,162.32,d751fd5edef97caed740093eabe1afa0
Lung_Opacity-1050.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1050.png,Lung_Opacity/images/Lung_Opacity-1050.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1050.png,Lung_Opacity/masks/Lung_Opacity-1050.png,True,False,299,299,L,131.97,82d7eb80e4dad6557e4f8ba59f0fa12e
Lung_Opacity-1051.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1051.png,Lung_Opacity/images/Lung_Opacity-1051.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1051.png,Lung_Opacity/masks/Lung_Opacity-1051.png,True,False,299,299,L,139.73,36e373483487bdba217d58ef67f8b39a
Lung_Opacity-1052.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1052.png,Lung_Opacity/images/Lung_Opacity-1052.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1052.png,Lung_Opacity/masks/Lung_Opacity-1052.png,True,False,299,299,L,109.83,e75d771ae451dc01697d1d35dd9d5bec
Lung_Opacity-1053.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1053.png,Lung_Opacity/images/Lung_Opacity-1053.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1053.png,Lung_Opacity/masks/Lung_Opacity-1053.png,True,False,299,299,L,135.68,3d57212ef57643a3ce75d8405ae869e6
Lung_Opacity-1054.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1054.png,Lung_Opacity/images/Lung_Opacity-1054.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1054.png,Lung_Opacity/masks/Lung_Opacity-1054.png,True,False,299,299,L,167.32,a15e79bc5ffad65c6170f26aa8cb6453
Lung_Opacity-1055.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1055.png,Lung_Opacity/images/Lung_Opacity-1055.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1055.png,Lung_Opacity/masks/Lung_Opacity-1055.png,True,False,299,299,L,148.35,9719b816d2c641ea6d2e1192c9087ac8
Lung_Opacity-1056.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1056.png,Lung_Opacity/images/Lung_Opacity-1056.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1056.png,Lung_Opacity/masks/Lung_Opacity-1056.png,True,False,299,299,L,166.96,5002f4e272575131712f3f62cbe04041
Lung_Opacity-1057.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1057.png,Lung_Opacity/images/Lung_Opacity-1057.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1057.png,Lung_Opacity/masks/Lung_Opacity-1057.png,True,False,299,299,L,115.51,0728ea2ac29f3a18fde1c5eb6d0955b6
Lung_Opacity-1058.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1058.png,Lung_Opacity/images/Lung_Opacity-1058.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1058.png,Lung_Opacity/masks/Lung_Opacity-1058.png,True,False,299,299,L,142.54,9574480dfa33c1ca6e7c09a02bf549ab
Lung_Opacity-1059.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1059.png,Lung_Opacity/images/Lung_Opacity-1059.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1059.png,Lung_Opacity/masks/Lung_Opacity-1059.png,True,False,299,299,L,96.05,e543633e58d7ee64fca1869239b8edb3
Lung_Opacity-106.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-106.png,Lung_Opacity/images/Lung_Opacity-106.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-106.png,Lung_Opacity/masks/Lung_Opacity-106.png,True,False,299,299,L,149.15,387506c691823c828a487b10355cc731
Lung_Opacity-1060.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1060.png,Lung_Opacity/images/Lung_Opacity-1060.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1060.png,Lung_Opacity/masks/Lung_Opacity-1060.png,True,False,299,299,L,83.14,fb5a5ba8793d4a8dcb3e66ca0e310d18
Lung_Opacity-1061.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1061.png,Lung_Opacity/images/Lung_Opacity-1061.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1061.png,Lung_Opacity/masks/Lung_Opacity-1061.png,True,False,299,299,L,119.14,3d6af5151931b2f24078b6063a476069
Lung_Opacity-1062.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1062.png,Lung_Opacity/images/Lung_Opacity-1062.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1062.png,Lung_Opacity/masks/Lung_Opacity-1062.png,True,False,299,299,L,110.48,6b59f03f0169d23f746cc0aa0c2c929c
Lung_Opacity-1063.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1063.png,Lung_Opacity/images/Lung_Opacity-1063.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1063.png,Lung_Opacity/masks/Lung_Opacity-1063.png,True,False,299,299,L,116.47,b99b2e8ba26e0044f1a3b04984b7324c
Lung_Opacity-1064.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1064.png,Lung_Opacity/images/Lung_Opacity-1064.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1064.png,Lung_Opacity/masks/Lung_Opacity-1064.png,True,False,299,299,L,118.84,226a013d8df294bf5c8b23b2d6e1b523
Lung_Opacity-1065.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1065.png,Lung_Opacity/images/Lung_Opacity-1065.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1065.png,Lung_Opacity/masks/Lung_Opacity-1065.png,True,False,299,299,L,175.97,a2f0d2a9fca8fedf3d59ee42b71864b7
Lung_Opacity-1066.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1066.png,Lung_Opacity/images/Lung_Opacity-1066.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1066.png,Lung_Opacity/masks/Lung_Opacity-1066.png,True,False,299,299,L,150.75,8c8345fba9ad0ba3ee1d12c4daabf222
Lung_Opacity-1067.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1067.png,Lung_Opacity/images/Lung_Opacity-1067.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1067.png,Lung_Opacity/masks/Lung_Opacity-1067.png,True,False,299,299,L,107.2,21499c4f3ac9112061d7968f8957b285
Lung_Opacity-1068.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1068.png,Lung_Opacity/images/Lung_Opacity-1068.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1068.png,Lung_Opacity/masks/Lung_Opacity-1068.png,True,False,299,299,L,123.44,30ce452f71626f3745f5ba67938d3ec2
Lung_Opacity-1069.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1069.png,Lung_Opacity/images/Lung_Opacity-1069.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1069.png,Lung_Opacity/masks/Lung_Opacity-1069.png,True,False,299,299,L,124.96,2dce42844733d0bcf57e6d915df82deb
Lung_Opacity-107.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-107.png,Lung_Opacity/images/Lung_Opacity-107.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-107.png,Lung_Opacity/masks/Lung_Opacity-107.png,True,False,299,299,L,94.06,78f8a68c069bf1cd9aa4aa3a423f0b5f
Lung_Opacity-1070.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1070.png,Lung_Opacity/images/Lung_Opacity-1070.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1070.png,Lung_Opacity/masks/Lung_Opacity-1070.png,True,False,299,299,L,102.87,0c0ae5f05014f6446bed9fd3b74d740b
Lung_Opacity-1071.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1071.png,Lung_Opacity/images/Lung_Opacity-1071.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1071.png,Lung_Opacity/masks/Lung_Opacity-1071.png,True,False,299,299,L,100.4,c00ee6ac779e50cb302b269045938eee
Lung_Opacity-1072.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1072.png,Lung_Opacity/images/Lung_Opacity-1072.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1072.png,Lung_Opacity/masks/Lung_Opacity-1072.png,True,False,299,299,L,153.43,05c32a67e90ba58c5ca03bada25db0be
Lung_Opacity-1073.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1073.png,Lung_Opacity/images/Lung_Opacity-1073.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1073.png,Lung_Opacity/masks/Lung_Opacity-1073.png,True,False,299,299,L,114.35,cbb35b68c282b3d1ba6ccc072d7aee7e
Lung_Opacity-1074.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1074.png,Lung_Opacity/images/Lung_Opacity-1074.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1074.png,Lung_Opacity/masks/Lung_Opacity-1074.png,True,False,299,299,L,95.62,e685dc81cea79a7865c288d980b7deb0
Lung_Opacity-1075.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1075.png,Lung_Opacity/images/Lung_Opacity-1075.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1075.png,Lung_Opacity/masks/Lung_Opacity-1075.png,True,False,299,299,L,119.91,85a0a0ddaee4dc31893c93ca52e684dc
Lung_Opacity-1076.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1076.png,Lung_Opacity/images/Lung_Opacity-1076.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1076.png,Lung_Opacity/masks/Lung_Opacity-1076.png,True,False,299,299,L,116.1,9a257e29907a088262b3ab12b260c6ca
Lung_Opacity-1077.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1077.png,Lung_Opacity/images/Lung_Opacity-1077.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1077.png,Lung_Opacity/masks/Lung_Opacity-1077.png,True,False,299,299,L,112.05,d8166b17da578d2a909b8d2882abb27e
Lung_Opacity-1078.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1078.png,Lung_Opacity/images/Lung_Opacity-1078.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1078.png,Lung_Opacity/masks/Lung_Opacity-1078.png,True,False,299,299,L,159.32,13dc176d51b866d725335dcd5acd751f
Lung_Opacity-1079.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1079.png,Lung_Opacity/images/Lung_Opacity-1079.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1079.png,Lung_Opacity/masks/Lung_Opacity-1079.png,True,False,299,299,L,105.9,d2532a258f481c9e7010d7249ba4147e
Lung_Opacity-108.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-108.png,Lung_Opacity/images/Lung_Opacity-108.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-108.png,Lung_Opacity/masks/Lung_Opacity-108.png,True,False,299,299,L,113.81,b13a1c6a00c20a85d3a2684ce02a6e59
Lung_Opacity-1080.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1080.png,Lung_Opacity/images/Lung_Opacity-1080.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1080.png,Lung_Opacity/masks/Lung_Opacity-1080.png,True,False,299,299,L,164.32,3edc7b3ee6684c77138781ee1bef09fc
Lung_Opacity-1081.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1081.png,Lung_Opacity/images/Lung_Opacity-1081.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1081.png,Lung_Opacity/masks/Lung_Opacity-1081.png,True,False,299,299,L,107.52,b97eedb74e10d886e322d8d0b76cf410
Lung_Opacity-1082.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1082.png,Lung_Opacity/images/Lung_Opacity-1082.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1082.png,Lung_Opacity/masks/Lung_Opacity-1082.png,True,False,299,299,L,107.64,cc1a548d4434854daa6c33f9e378cee7
Lung_Opacity-1083.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1083.png,Lung_Opacity/images/Lung_Opacity-1083.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1083.png,Lung_Opacity/masks/Lung_Opacity-1083.png,True,False,299,299,L,100.11,1e04b089cc37ae007ea343d0653aad18
Lung_Opacity-1084.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1084.png,Lung_Opacity/images/Lung_Opacity-1084.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1084.png,Lung_Opacity/masks/Lung_Opacity-1084.png,True,False,299,299,L,130.22,73858d3a6fb5b31ff7630648f9f68339
Lung_Opacity-1085.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1085.png,Lung_Opacity/images/Lung_Opacity-1085.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1085.png,Lung_Opacity/masks/Lung_Opacity-1085.png,True,False,299,299,L,117.69,e3d0c2b7f6ea551afdad2588015fd6cb
Lung_Opacity-1086.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1086.png,Lung_Opacity/images/Lung_Opacity-1086.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1086.png,Lung_Opacity/masks/Lung_Opacity-1086.png,True,False,299,299,L,119.49,bc285af970b368fd54cd4d445437e401
Lung_Opacity-1087.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1087.png,Lung_Opacity/images/Lung_Opacity-1087.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1087.png,Lung_Opacity/masks/Lung_Opacity-1087.png,True,False,299,299,L,142.49,e4aae84b828f5552e75c8b24047d4b90
Lung_Opacity-1088.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1088.png,Lung_Opacity/images/Lung_Opacity-1088.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1088.png,Lung_Opacity/masks/Lung_Opacity-1088.png,True,False,299,299,L,112.15,b069c12bb9ae004539f4fdf06667330c
Lung_Opacity-1089.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1089.png,Lung_Opacity/images/Lung_Opacity-1089.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1089.png,Lung_Opacity/masks/Lung_Opacity-1089.png,True,False,299,299,L,125.04,bb218ae2f432bc4f1f1f83e560ed27d3
Lung_Opacity-109.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-109.png,Lung_Opacity/images/Lung_Opacity-109.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-109.png,Lung_Opacity/masks/Lung_Opacity-109.png,True,False,299,299,L,104.99,3aa26105d79dfe30ae2b4c04b3a49542
Lung_Opacity-1090.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1090.png,Lung_Opacity/images/Lung_Opacity-1090.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1090.png,Lung_Opacity/masks/Lung_Opacity-1090.png,True,False,299,299,L,103.91,eeb3c5a3e9be81b76f84b3cf25f95648
Lung_Opacity-1091.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1091.png,Lung_Opacity/images/Lung_Opacity-1091.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1091.png,Lung_Opacity/masks/Lung_Opacity-1091.png,True,False,299,299,L,107.28,6e85ec3d184d4ab40075dcef3e27c9f3
Lung_Opacity-1092.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1092.png,Lung_Opacity/images/Lung_Opacity-1092.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1092.png,Lung_Opacity/masks/Lung_Opacity-1092.png,True,False,299,299,L,111.7,0fbb02f74e8a39f3939af6fea80fa12a
Lung_Opacity-1093.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1093.png,Lung_Opacity/images/Lung_Opacity-1093.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1093.png,Lung_Opacity/masks/Lung_Opacity-1093.png,True,False,299,299,L,115.98,7296fa146f77887d080996beb70d635e
Lung_Opacity-1094.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1094.png,Lung_Opacity/images/Lung_Opacity-1094.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1094.png,Lung_Opacity/masks/Lung_Opacity-1094.png,True,False,299,299,L,112.89,9728c71f7823ec1b664c1f7fe9119be2
Lung_Opacity-1095.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1095.png,Lung_Opacity/images/Lung_Opacity-1095.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1095.png,Lung_Opacity/masks/Lung_Opacity-1095.png,True,False,299,299,L,146.66,29a093857d4c3267b195d00055f41a28
Lung_Opacity-1096.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1096.png,Lung_Opacity/images/Lung_Opacity-1096.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1096.png,Lung_Opacity/masks/Lung_Opacity-1096.png,True,False,299,299,L,120.51,4f36c99cc0d597b195038e407d4ee041
Lung_Opacity-1097.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1097.png,Lung_Opacity/images/Lung_Opacity-1097.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1097.png,Lung_Opacity/masks/Lung_Opacity-1097.png,True,False,299,299,L,167.1,d50dd8f10150c49979b78d7f420090ac
Lung_Opacity-1098.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1098.png,Lung_Opacity/images/Lung_Opacity-1098.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1098.png,Lung_Opacity/masks/Lung_Opacity-1098.png,True,False,299,299,L,177.67,093a523c93d54ecee9bd8d91a5de0b1d
Lung_Opacity-1099.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1099.png,Lung_Opacity/images/Lung_Opacity-1099.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1099.png,Lung_Opacity/masks/Lung_Opacity-1099.png,True,False,299,299,L,132.41,d6bb2f833b3dbf9a8de75c2be7c0c4f1
Lung_Opacity-11.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-11.png,Lung_Opacity/images/Lung_Opacity-11.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-11.png,Lung_Opacity/masks/Lung_Opacity-11.png,True,False,299,299,L,179.17,0346617aab11b6cb8fc1c22167d9c360
Lung_Opacity-110.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-110.png,Lung_Opacity/images/Lung_Opacity-110.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-110.png,Lung_Opacity/masks/Lung_Opacity-110.png,True,False,299,299,L,124.75,ab27a0884acee29f5cb266dc8f6e81e6
Lung_Opacity-1100.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1100.png,Lung_Opacity/images/Lung_Opacity-1100.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1100.png,Lung_Opacity/masks/Lung_Opacity-1100.png,True,False,299,299,L,86.61,18e9af23e742a7f09a02542c278acd49
Lung_Opacity-1101.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1101.png,Lung_Opacity/images/Lung_Opacity-1101.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1101.png,Lung_Opacity/masks/Lung_Opacity-1101.png,True,False,299,299,L,73.08,fbe22f27532a47664a4bbaa47fb40f74
Lung_Opacity-1102.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1102.png,Lung_Opacity/images/Lung_Opacity-1102.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1102.png,Lung_Opacity/masks/Lung_Opacity-1102.png,True,False,299,299,L,116.54,a0acd1c9cb2a7ef42067cce47998189a
Lung_Opacity-1103.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1103.png,Lung_Opacity/images/Lung_Opacity-1103.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1103.png,Lung_Opacity/masks/Lung_Opacity-1103.png,True,False,299,299,L,107.91,ea10d316e368d4d3b3017e4b21ce20fb
Lung_Opacity-1104.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1104.png,Lung_Opacity/images/Lung_Opacity-1104.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1104.png,Lung_Opacity/masks/Lung_Opacity-1104.png,True,False,299,299,L,124.17,5c58e49d1db81754bdcf65185846a348
Lung_Opacity-1105.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1105.png,Lung_Opacity/images/Lung_Opacity-1105.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1105.png,Lung_Opacity/masks/Lung_Opacity-1105.png,True,False,299,299,L,112.29,bd8cc82f874e4dda9af56a2544bf80a2
Lung_Opacity-1106.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1106.png,Lung_Opacity/images/Lung_Opacity-1106.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1106.png,Lung_Opacity/masks/Lung_Opacity-1106.png,True,False,299,299,L,124.61,1b5f8467b97f654c99fabfd0c3713562
Lung_Opacity-1107.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1107.png,Lung_Opacity/images/Lung_Opacity-1107.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1107.png,Lung_Opacity/masks/Lung_Opacity-1107.png,True,False,299,299,L,89.97,004544f1db7baa903258390d4d967aed
Lung_Opacity-1108.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1108.png,Lung_Opacity/images/Lung_Opacity-1108.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1108.png,Lung_Opacity/masks/Lung_Opacity-1108.png,True,False,299,299,L,151.22,32451299f29788017eed98dd4b1ab77e
Lung_Opacity-1109.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1109.png,Lung_Opacity/images/Lung_Opacity-1109.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1109.png,Lung_Opacity/masks/Lung_Opacity-1109.png,True,False,299,299,L,104.76,f7b7d17db54bfa62427ba7e647c28d44
Lung_Opacity-111.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-111.png,Lung_Opacity/images/Lung_Opacity-111.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-111.png,Lung_Opacity/masks/Lung_Opacity-111.png,True,False,299,299,L,138.52,d674ca3cabc40345dcc309e3d803bd49
Lung_Opacity-1110.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1110.png,Lung_Opacity/images/Lung_Opacity-1110.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1110.png,Lung_Opacity/masks/Lung_Opacity-1110.png,True,False,299,299,L,154.75,6aebfe615dd7793d7eb4af6321f021f9
Lung_Opacity-1111.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1111.png,Lung_Opacity/images/Lung_Opacity-1111.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1111.png,Lung_Opacity/masks/Lung_Opacity-1111.png,True,False,299,299,L,130.81,2cf81a34a384501753e30458373744cc
Lung_Opacity-1112.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1112.png,Lung_Opacity/images/Lung_Opacity-1112.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1112.png,Lung_Opacity/masks/Lung_Opacity-1112.png,True,False,299,299,L,127.51,5d9bb24735d9fe2976715a6e0cb0407f
Lung_Opacity-1113.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1113.png,Lung_Opacity/images/Lung_Opacity-1113.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1113.png,Lung_Opacity/masks/Lung_Opacity-1113.png,True,False,299,299,L,89.6,53ce04d93fc0183e45fe8c5ae63444e3
Lung_Opacity-1114.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1114.png,Lung_Opacity/images/Lung_Opacity-1114.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1114.png,Lung_Opacity/masks/Lung_Opacity-1114.png,True,False,299,299,L,86.85,24b2d4359c2d2ff23e72e9b09fba7e6d
Lung_Opacity-1115.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1115.png,Lung_Opacity/images/Lung_Opacity-1115.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1115.png,Lung_Opacity/masks/Lung_Opacity-1115.png,True,False,299,299,L,108.68,4c8b4ada1b4204f2907589e2fc3fb30b
Lung_Opacity-1116.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1116.png,Lung_Opacity/images/Lung_Opacity-1116.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1116.png,Lung_Opacity/masks/Lung_Opacity-1116.png,True,False,299,299,L,154.57,ad8f9339a82fd400befbae81803a9bb5
Lung_Opacity-1117.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1117.png,Lung_Opacity/images/Lung_Opacity-1117.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1117.png,Lung_Opacity/masks/Lung_Opacity-1117.png,True,False,299,299,L,127.41,3ac9614b45a7645340656d0ec8f68a15
Lung_Opacity-1118.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1118.png,Lung_Opacity/images/Lung_Opacity-1118.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1118.png,Lung_Opacity/masks/Lung_Opacity-1118.png,True,False,299,299,L,107.98,be6f3b0eb0c67c1ad3844036ba852548
Lung_Opacity-1119.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1119.png,Lung_Opacity/images/Lung_Opacity-1119.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1119.png,Lung_Opacity/masks/Lung_Opacity-1119.png,True,False,299,299,L,162.24,1f4c6919713b6447022615a1b57b67be
Lung_Opacity-112.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-112.png,Lung_Opacity/images/Lung_Opacity-112.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-112.png,Lung_Opacity/masks/Lung_Opacity-112.png,True,False,299,299,L,103.73,10fdbdd5771ee7f8fdc0c5bce65f1775
Lung_Opacity-1120.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1120.png,Lung_Opacity/images/Lung_Opacity-1120.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1120.png,Lung_Opacity/masks/Lung_Opacity-1120.png,True,False,299,299,L,156.87,08db8ea7971739d08c357692d06a49fa
Lung_Opacity-1121.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1121.png,Lung_Opacity/images/Lung_Opacity-1121.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1121.png,Lung_Opacity/masks/Lung_Opacity-1121.png,True,False,299,299,L,118.02,af6c2282a6e2a7ab52ae971ea5115c1a
Lung_Opacity-1122.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1122.png,Lung_Opacity/images/Lung_Opacity-1122.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1122.png,Lung_Opacity/masks/Lung_Opacity-1122.png,True,False,299,299,L,116.58,b88492e9bc9f147cb502811754238635
Lung_Opacity-1123.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1123.png,Lung_Opacity/images/Lung_Opacity-1123.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1123.png,Lung_Opacity/masks/Lung_Opacity-1123.png,True,False,299,299,L,154.65,8c646d9888ade3379dae1c3757967091
Lung_Opacity-1124.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1124.png,Lung_Opacity/images/Lung_Opacity-1124.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1124.png,Lung_Opacity/masks/Lung_Opacity-1124.png,True,False,299,299,L,120.04,2a0601dd2bcc9a7c7851c950097a1831
Lung_Opacity-1125.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1125.png,Lung_Opacity/images/Lung_Opacity-1125.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1125.png,Lung_Opacity/masks/Lung_Opacity-1125.png,True,False,299,299,L,115.51,335e78a05f5ec4619b4b91825ade5eeb
Lung_Opacity-1126.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1126.png,Lung_Opacity/images/Lung_Opacity-1126.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1126.png,Lung_Opacity/masks/Lung_Opacity-1126.png,True,False,299,299,L,130.9,fc5aa5bfca22faab9346142eb4604361
Lung_Opacity-1127.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1127.png,Lung_Opacity/images/Lung_Opacity-1127.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1127.png,Lung_Opacity/masks/Lung_Opacity-1127.png,True,False,299,299,L,103.99,0119f3dc11841c1cc0d406728aefa063
Lung_Opacity-1128.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1128.png,Lung_Opacity/images/Lung_Opacity-1128.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1128.png,Lung_Opacity/masks/Lung_Opacity-1128.png,True,False,299,299,L,171.93,f0c740e84206ef5394f21d45bccbed6b
Lung_Opacity-1129.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1129.png,Lung_Opacity/images/Lung_Opacity-1129.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1129.png,Lung_Opacity/masks/Lung_Opacity-1129.png,True,False,299,299,L,127.61,f1ac527c8f665ae4f37276db33997632
Lung_Opacity-113.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-113.png,Lung_Opacity/images/Lung_Opacity-113.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-113.png,Lung_Opacity/masks/Lung_Opacity-113.png,True,False,299,299,L,95.38,641002240412a4664d7f5eaabb938543
Lung_Opacity-1130.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1130.png,Lung_Opacity/images/Lung_Opacity-1130.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1130.png,Lung_Opacity/masks/Lung_Opacity-1130.png,True,False,299,299,L,113.36,61bc311b2a34f07a97850d7f64a60d10
Lung_Opacity-1131.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1131.png,Lung_Opacity/images/Lung_Opacity-1131.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1131.png,Lung_Opacity/masks/Lung_Opacity-1131.png,True,False,299,299,L,111.77,0dd03a2a256ceed0cb311da8eadae1b1
Lung_Opacity-1132.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1132.png,Lung_Opacity/images/Lung_Opacity-1132.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1132.png,Lung_Opacity/masks/Lung_Opacity-1132.png,True,False,299,299,L,105.28,f029ef5ba678739dfde98a7d4393cf6f
Lung_Opacity-1133.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1133.png,Lung_Opacity/images/Lung_Opacity-1133.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1133.png,Lung_Opacity/masks/Lung_Opacity-1133.png,True,False,299,299,L,89.5,0e4e5d1966d7ddcaf51820460378aece
Lung_Opacity-1134.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1134.png,Lung_Opacity/images/Lung_Opacity-1134.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1134.png,Lung_Opacity/masks/Lung_Opacity-1134.png,True,False,299,299,L,149.52,260de84cc81b92575bc15b4dc7f7a98d
Lung_Opacity-1135.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1135.png,Lung_Opacity/images/Lung_Opacity-1135.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1135.png,Lung_Opacity/masks/Lung_Opacity-1135.png,True,False,299,299,L,125.49,e5a508a79f70ba9504aa4cb8024c9464
Lung_Opacity-1136.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1136.png,Lung_Opacity/images/Lung_Opacity-1136.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1136.png,Lung_Opacity/masks/Lung_Opacity-1136.png,True,False,299,299,L,113.91,9e6a9d5a4ccf768149919d9854b88049
Lung_Opacity-1137.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1137.png,Lung_Opacity/images/Lung_Opacity-1137.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1137.png,Lung_Opacity/masks/Lung_Opacity-1137.png,True,False,299,299,L,139.95,1a103ee17cd8f52a3f6654101b7c3f3c
Lung_Opacity-1138.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1138.png,Lung_Opacity/images/Lung_Opacity-1138.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1138.png,Lung_Opacity/masks/Lung_Opacity-1138.png,True,False,299,299,L,149.08,e5cf4be3291db757cd707fdfa113da21
Lung_Opacity-1139.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1139.png,Lung_Opacity/images/Lung_Opacity-1139.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1139.png,Lung_Opacity/masks/Lung_Opacity-1139.png,True,False,299,299,L,120.26,79dd4c21d73c51c5f9ecd2d9f95b212f
Lung_Opacity-114.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-114.png,Lung_Opacity/images/Lung_Opacity-114.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-114.png,Lung_Opacity/masks/Lung_Opacity-114.png,True,False,299,299,L,114.1,dc080b7579289ce9d5ab24b92eca796d
Lung_Opacity-1140.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1140.png,Lung_Opacity/images/Lung_Opacity-1140.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1140.png,Lung_Opacity/masks/Lung_Opacity-1140.png,True,False,299,299,L,114.93,b3ec0ad957fb2ebb87dfef6ad4470e62
Lung_Opacity-1141.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1141.png,Lung_Opacity/images/Lung_Opacity-1141.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1141.png,Lung_Opacity/masks/Lung_Opacity-1141.png,True,False,299,299,L,120.62,ef0863599e0785e02d6418df8a54e46d
Lung_Opacity-1142.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1142.png,Lung_Opacity/images/Lung_Opacity-1142.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1142.png,Lung_Opacity/masks/Lung_Opacity-1142.png,True,False,299,299,L,118.0,6de5ce8bb7e23b812eb2f0347176ecf5
Lung_Opacity-1143.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1143.png,Lung_Opacity/images/Lung_Opacity-1143.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1143.png,Lung_Opacity/masks/Lung_Opacity-1143.png,True,False,299,299,L,52.06,08f5e89e79b4f1e45cacb3b79b856158
Lung_Opacity-1144.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1144.png,Lung_Opacity/images/Lung_Opacity-1144.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1144.png,Lung_Opacity/masks/Lung_Opacity-1144.png,True,False,299,299,L,46.97,ebb99f8f6c1b2075be055c0572be8356
Lung_Opacity-1145.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1145.png,Lung_Opacity/images/Lung_Opacity-1145.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1145.png,Lung_Opacity/masks/Lung_Opacity-1145.png,True,False,299,299,L,91.06,a79063422a000118faeabe6003f37b3d
Lung_Opacity-1146.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1146.png,Lung_Opacity/images/Lung_Opacity-1146.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1146.png,Lung_Opacity/masks/Lung_Opacity-1146.png,True,False,299,299,L,146.71,eb8fc0f1f43ddab0e0504bdbf28635de
Lung_Opacity-1147.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1147.png,Lung_Opacity/images/Lung_Opacity-1147.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1147.png,Lung_Opacity/masks/Lung_Opacity-1147.png,True,False,299,299,L,131.76,4f5bf6dbb0cd34149d9c89263e44132b
Lung_Opacity-1148.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1148.png,Lung_Opacity/images/Lung_Opacity-1148.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1148.png,Lung_Opacity/masks/Lung_Opacity-1148.png,True,False,299,299,L,149.46,eee5768e1c122aa87a9e48892c097579
Lung_Opacity-1149.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1149.png,Lung_Opacity/images/Lung_Opacity-1149.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1149.png,Lung_Opacity/masks/Lung_Opacity-1149.png,True,False,299,299,L,109.81,2832cc76e3e453e85c5c86bc309a6bf5
Lung_Opacity-115.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-115.png,Lung_Opacity/images/Lung_Opacity-115.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-115.png,Lung_Opacity/masks/Lung_Opacity-115.png,True,False,299,299,L,155.73,674d2d3480a86edda0e323ffe51379d4
Lung_Opacity-1150.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1150.png,Lung_Opacity/images/Lung_Opacity-1150.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1150.png,Lung_Opacity/masks/Lung_Opacity-1150.png,True,False,299,299,L,96.17,46eb4d368ebe8ec73ef6000815ef0c55
Lung_Opacity-1151.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1151.png,Lung_Opacity/images/Lung_Opacity-1151.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1151.png,Lung_Opacity/masks/Lung_Opacity-1151.png,True,False,299,299,L,177.55,dfa61ce354efc2a210bb5121ca4de032
Lung_Opacity-1152.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1152.png,Lung_Opacity/images/Lung_Opacity-1152.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1152.png,Lung_Opacity/masks/Lung_Opacity-1152.png,True,False,299,299,L,112.54,e0b2e7561b680d53eaebc7916407de87
Lung_Opacity-1153.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1153.png,Lung_Opacity/images/Lung_Opacity-1153.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1153.png,Lung_Opacity/masks/Lung_Opacity-1153.png,True,False,299,299,L,86.8,abe971b7b3c064a09c823619992609fb
Lung_Opacity-1154.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1154.png,Lung_Opacity/images/Lung_Opacity-1154.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1154.png,Lung_Opacity/masks/Lung_Opacity-1154.png,True,False,299,299,L,147.59,bb4c8e6943a1d7368bdb2c0d868d2a05
Lung_Opacity-1155.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1155.png,Lung_Opacity/images/Lung_Opacity-1155.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1155.png,Lung_Opacity/masks/Lung_Opacity-1155.png,True,False,299,299,L,112.21,2ae7fc5ad850bfff82e7892a57492e2c
Lung_Opacity-1156.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1156.png,Lung_Opacity/images/Lung_Opacity-1156.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1156.png,Lung_Opacity/masks/Lung_Opacity-1156.png,True,False,299,299,L,111.84,b575f465d8cb644ed528b708ce82d463
Lung_Opacity-1157.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1157.png,Lung_Opacity/images/Lung_Opacity-1157.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1157.png,Lung_Opacity/masks/Lung_Opacity-1157.png,True,False,299,299,L,110.16,91368d3a50b646265d521130c4e395f1
Lung_Opacity-1158.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1158.png,Lung_Opacity/images/Lung_Opacity-1158.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1158.png,Lung_Opacity/masks/Lung_Opacity-1158.png,True,False,299,299,L,133.77,5ae1399ab9ac912f1abf4bda444f0901
Lung_Opacity-1159.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1159.png,Lung_Opacity/images/Lung_Opacity-1159.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1159.png,Lung_Opacity/masks/Lung_Opacity-1159.png,True,False,299,299,L,109.29,5465670d37692274e2fe05e5bd72f805
Lung_Opacity-116.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-116.png,Lung_Opacity/images/Lung_Opacity-116.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-116.png,Lung_Opacity/masks/Lung_Opacity-116.png,True,False,299,299,L,115.12,1d5b57fbc2493c9982cc5962fe3bfa17
Lung_Opacity-1160.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1160.png,Lung_Opacity/images/Lung_Opacity-1160.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1160.png,Lung_Opacity/masks/Lung_Opacity-1160.png,True,False,299,299,L,131.58,3d987ac7017a0cf0490f15e963a5a4cc
Lung_Opacity-1161.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1161.png,Lung_Opacity/images/Lung_Opacity-1161.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1161.png,Lung_Opacity/masks/Lung_Opacity-1161.png,True,False,299,299,L,165.73,a889dae652d50fae6aa610757b66a075
Lung_Opacity-1162.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1162.png,Lung_Opacity/images/Lung_Opacity-1162.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1162.png,Lung_Opacity/masks/Lung_Opacity-1162.png,True,False,299,299,L,164.74,0dae7f50b0c6d245685ec56af8d779a5
Lung_Opacity-1163.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1163.png,Lung_Opacity/images/Lung_Opacity-1163.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1163.png,Lung_Opacity/masks/Lung_Opacity-1163.png,True,False,299,299,L,122.08,b38b282c6407de784ab6e1a19e4aa54f
Lung_Opacity-1164.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1164.png,Lung_Opacity/images/Lung_Opacity-1164.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1164.png,Lung_Opacity/masks/Lung_Opacity-1164.png,True,False,299,299,L,138.46,5c3a416101695cfbf37ddcbbcd370b72
Lung_Opacity-1165.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1165.png,Lung_Opacity/images/Lung_Opacity-1165.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1165.png,Lung_Opacity/masks/Lung_Opacity-1165.png,True,False,299,299,L,120.39,72916cb74d355d50b380a4a9e121a538
Lung_Opacity-1166.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1166.png,Lung_Opacity/images/Lung_Opacity-1166.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1166.png,Lung_Opacity/masks/Lung_Opacity-1166.png,True,False,299,299,L,121.47,3473a0c3f841a7c8d94f06e23f508129
Lung_Opacity-1167.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1167.png,Lung_Opacity/images/Lung_Opacity-1167.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1167.png,Lung_Opacity/masks/Lung_Opacity-1167.png,True,False,299,299,L,119.35,e65aa169318e02b8e249463209123b1f
Lung_Opacity-1168.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1168.png,Lung_Opacity/images/Lung_Opacity-1168.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1168.png,Lung_Opacity/masks/Lung_Opacity-1168.png,True,False,299,299,L,140.86,0ef51de132d1abe59a73766042015684
Lung_Opacity-1169.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1169.png,Lung_Opacity/images/Lung_Opacity-1169.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1169.png,Lung_Opacity/masks/Lung_Opacity-1169.png,True,False,299,299,L,116.42,19856e482013fbb1a468589a8df6b049
Lung_Opacity-117.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-117.png,Lung_Opacity/images/Lung_Opacity-117.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-117.png,Lung_Opacity/masks/Lung_Opacity-117.png,True,False,299,299,L,127.29,d7e59ab12984ed86825e0bb11988bd9b
Lung_Opacity-1170.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1170.png,Lung_Opacity/images/Lung_Opacity-1170.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1170.png,Lung_Opacity/masks/Lung_Opacity-1170.png,True,False,299,299,L,110.86,d306c5cc74d90482c335323d0b5aef80
Lung_Opacity-1171.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1171.png,Lung_Opacity/images/Lung_Opacity-1171.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1171.png,Lung_Opacity/masks/Lung_Opacity-1171.png,True,False,299,299,L,71.04,eaca97736e81867e8b914fce3394009b
Lung_Opacity-1172.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1172.png,Lung_Opacity/images/Lung_Opacity-1172.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1172.png,Lung_Opacity/masks/Lung_Opacity-1172.png,True,False,299,299,L,114.87,6f5ee9ba8ae8f869525fae8ca310c81a
Lung_Opacity-1173.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1173.png,Lung_Opacity/images/Lung_Opacity-1173.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1173.png,Lung_Opacity/masks/Lung_Opacity-1173.png,True,False,299,299,L,131.89,b409ba2dcfad9a6bd394e29cb69c74a6
Lung_Opacity-1174.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1174.png,Lung_Opacity/images/Lung_Opacity-1174.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1174.png,Lung_Opacity/masks/Lung_Opacity-1174.png,True,False,299,299,L,128.2,9af8cfa5c9aad06b6af738726c19b566
Lung_Opacity-1175.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1175.png,Lung_Opacity/images/Lung_Opacity-1175.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1175.png,Lung_Opacity/masks/Lung_Opacity-1175.png,True,False,299,299,L,111.91,453b038b266811527e57f768e95f73e6
Lung_Opacity-1176.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1176.png,Lung_Opacity/images/Lung_Opacity-1176.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1176.png,Lung_Opacity/masks/Lung_Opacity-1176.png,True,False,299,299,L,118.19,074fb76be97e4fc698222a74d3151621
Lung_Opacity-1177.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1177.png,Lung_Opacity/images/Lung_Opacity-1177.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1177.png,Lung_Opacity/masks/Lung_Opacity-1177.png,True,False,299,299,L,136.67,67ca421a3384bd0d2c5d8de75cc29d46
Lung_Opacity-1178.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1178.png,Lung_Opacity/images/Lung_Opacity-1178.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1178.png,Lung_Opacity/masks/Lung_Opacity-1178.png,True,False,299,299,L,136.91,34e96749cfb78d273a182a084d0f6140
Lung_Opacity-1179.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1179.png,Lung_Opacity/images/Lung_Opacity-1179.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1179.png,Lung_Opacity/masks/Lung_Opacity-1179.png,True,False,299,299,L,108.55,e596b2b3b0cbceffd35e910e8a522c26
Lung_Opacity-118.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-118.png,Lung_Opacity/images/Lung_Opacity-118.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-118.png,Lung_Opacity/masks/Lung_Opacity-118.png,True,False,299,299,L,121.17,5d8b879214bc3243f7bd737d9a23bfd4
Lung_Opacity-1180.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1180.png,Lung_Opacity/images/Lung_Opacity-1180.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1180.png,Lung_Opacity/masks/Lung_Opacity-1180.png,True,False,299,299,L,140.02,cb8dce33ca03cfc04337bbd1eee2aa08
Lung_Opacity-1181.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1181.png,Lung_Opacity/images/Lung_Opacity-1181.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1181.png,Lung_Opacity/masks/Lung_Opacity-1181.png,True,False,299,299,L,95.91,c8b3b2f3929dedd7ec7cf1e3bf08e3d6
Lung_Opacity-1182.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1182.png,Lung_Opacity/images/Lung_Opacity-1182.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1182.png,Lung_Opacity/masks/Lung_Opacity-1182.png,True,False,299,299,L,99.36,02be748a328e7e285dc73f4c3a726e0b
Lung_Opacity-1183.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1183.png,Lung_Opacity/images/Lung_Opacity-1183.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1183.png,Lung_Opacity/masks/Lung_Opacity-1183.png,True,False,299,299,L,121.79,e610dfa3d9c1a24dad62fe374700451f
Lung_Opacity-1184.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1184.png,Lung_Opacity/images/Lung_Opacity-1184.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1184.png,Lung_Opacity/masks/Lung_Opacity-1184.png,True,False,299,299,L,109.27,4c355085c383c9de046c1dab01514154
Lung_Opacity-1185.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1185.png,Lung_Opacity/images/Lung_Opacity-1185.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1185.png,Lung_Opacity/masks/Lung_Opacity-1185.png,True,False,299,299,L,124.9,de1e64f87cdf2747859460977a891834
Lung_Opacity-1186.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1186.png,Lung_Opacity/images/Lung_Opacity-1186.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1186.png,Lung_Opacity/masks/Lung_Opacity-1186.png,True,False,299,299,L,156.56,6d48103cad5aa7604cdedef954bdd7f8
Lung_Opacity-1187.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1187.png,Lung_Opacity/images/Lung_Opacity-1187.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1187.png,Lung_Opacity/masks/Lung_Opacity-1187.png,True,False,299,299,L,153.57,4b00aa99e7a01fee2505520a101f16f8
Lung_Opacity-1188.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1188.png,Lung_Opacity/images/Lung_Opacity-1188.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1188.png,Lung_Opacity/masks/Lung_Opacity-1188.png,True,False,299,299,L,123.49,0b1806231ad0549884930b96cdff7c31
Lung_Opacity-1189.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1189.png,Lung_Opacity/images/Lung_Opacity-1189.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1189.png,Lung_Opacity/masks/Lung_Opacity-1189.png,True,False,299,299,L,112.77,8e6fbccdf28016da9ff5fb03b1baf1c4
Lung_Opacity-119.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-119.png,Lung_Opacity/images/Lung_Opacity-119.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-119.png,Lung_Opacity/masks/Lung_Opacity-119.png,True,False,299,299,L,147.42,1fb72462c7ee5590f53d681fb4653ef2
Lung_Opacity-1190.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1190.png,Lung_Opacity/images/Lung_Opacity-1190.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1190.png,Lung_Opacity/masks/Lung_Opacity-1190.png,True,False,299,299,L,153.31,95b9aae87c9960ab274906d406d71a3f
Lung_Opacity-1191.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1191.png,Lung_Opacity/images/Lung_Opacity-1191.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1191.png,Lung_Opacity/masks/Lung_Opacity-1191.png,True,False,299,299,L,115.43,da66661ab7539945fd368acf460418f0
Lung_Opacity-1192.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1192.png,Lung_Opacity/images/Lung_Opacity-1192.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1192.png,Lung_Opacity/masks/Lung_Opacity-1192.png,True,False,299,299,L,166.64,777438629eb03497881275b4d2748683
Lung_Opacity-1193.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1193.png,Lung_Opacity/images/Lung_Opacity-1193.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1193.png,Lung_Opacity/masks/Lung_Opacity-1193.png,True,False,299,299,L,157.45,345f7432148def3a1bdd7d19b762a323
Lung_Opacity-1194.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1194.png,Lung_Opacity/images/Lung_Opacity-1194.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1194.png,Lung_Opacity/masks/Lung_Opacity-1194.png,True,False,299,299,L,104.12,fefe731aaa7701ecbe6b28dc188d6a8b
Lung_Opacity-1195.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1195.png,Lung_Opacity/images/Lung_Opacity-1195.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1195.png,Lung_Opacity/masks/Lung_Opacity-1195.png,True,False,299,299,L,95.78,04a698a44af7cdd9a1639e46231913ab
Lung_Opacity-1196.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1196.png,Lung_Opacity/images/Lung_Opacity-1196.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1196.png,Lung_Opacity/masks/Lung_Opacity-1196.png,True,False,299,299,L,108.1,bc94ccb4dbbcfd11c51d852526a730e2
Lung_Opacity-1197.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1197.png,Lung_Opacity/images/Lung_Opacity-1197.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1197.png,Lung_Opacity/masks/Lung_Opacity-1197.png,True,False,299,299,L,113.33,85e429a300e4f4e2feab5bff534f4e4b
Lung_Opacity-1198.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1198.png,Lung_Opacity/images/Lung_Opacity-1198.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1198.png,Lung_Opacity/masks/Lung_Opacity-1198.png,True,False,299,299,L,107.67,4cb73fc58cb9efcaf642dba8b89a132f
Lung_Opacity-1199.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1199.png,Lung_Opacity/images/Lung_Opacity-1199.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1199.png,Lung_Opacity/masks/Lung_Opacity-1199.png,True,False,299,299,L,163.94,19091e07ac33bee156a85321d6daa91b
Lung_Opacity-12.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-12.png,Lung_Opacity/images/Lung_Opacity-12.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-12.png,Lung_Opacity/masks/Lung_Opacity-12.png,True,False,299,299,L,121.97,e7cd25a7923e93efb13c9d656ade0894
Lung_Opacity-120.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-120.png,Lung_Opacity/images/Lung_Opacity-120.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-120.png,Lung_Opacity/masks/Lung_Opacity-120.png,True,False,299,299,L,161.25,052a49caeb86f6e1f4d83e6f3068345c
Lung_Opacity-1200.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1200.png,Lung_Opacity/images/Lung_Opacity-1200.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1200.png,Lung_Opacity/masks/Lung_Opacity-1200.png,True,False,299,299,L,172.89,2a78c431631429fe47c0f6bd8d25bc9b
Lung_Opacity-1201.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1201.png,Lung_Opacity/images/Lung_Opacity-1201.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1201.png,Lung_Opacity/masks/Lung_Opacity-1201.png,True,False,299,299,L,138.65,cf33cf120546258357aefa0dbb6386eb
Lung_Opacity-1202.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1202.png,Lung_Opacity/images/Lung_Opacity-1202.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1202.png,Lung_Opacity/masks/Lung_Opacity-1202.png,True,False,299,299,L,153.05,42ade98c65ab66c9041c90518acb2ef7
Lung_Opacity-1203.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1203.png,Lung_Opacity/images/Lung_Opacity-1203.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1203.png,Lung_Opacity/masks/Lung_Opacity-1203.png,True,False,299,299,L,128.3,b40ee6e3879b10a83c5c072febf4a0bb
Lung_Opacity-1204.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1204.png,Lung_Opacity/images/Lung_Opacity-1204.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1204.png,Lung_Opacity/masks/Lung_Opacity-1204.png,True,False,299,299,L,121.04,9ff7fbd6114273bcce427ecfa5ccf1db
Lung_Opacity-1205.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1205.png,Lung_Opacity/images/Lung_Opacity-1205.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1205.png,Lung_Opacity/masks/Lung_Opacity-1205.png,True,False,299,299,L,124.38,7b8066ff143ff9fdd6c48c77cfb9161c
Lung_Opacity-1206.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1206.png,Lung_Opacity/images/Lung_Opacity-1206.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1206.png,Lung_Opacity/masks/Lung_Opacity-1206.png,True,False,299,299,L,160.04,44a844ae298b3864bb5f959cf0b05b80
Lung_Opacity-1207.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1207.png,Lung_Opacity/images/Lung_Opacity-1207.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1207.png,Lung_Opacity/masks/Lung_Opacity-1207.png,True,False,299,299,L,110.57,ebe4d294c84f2341bc2874f2e2c216fc
Lung_Opacity-1208.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1208.png,Lung_Opacity/images/Lung_Opacity-1208.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1208.png,Lung_Opacity/masks/Lung_Opacity-1208.png,True,False,299,299,L,158.99,100f992153f6c59f590ebdad229f58c7
Lung_Opacity-1209.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1209.png,Lung_Opacity/images/Lung_Opacity-1209.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1209.png,Lung_Opacity/masks/Lung_Opacity-1209.png,True,False,299,299,L,129.7,c2f0e8683182dfa2bc37f261a4c87b62
Lung_Opacity-121.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-121.png,Lung_Opacity/images/Lung_Opacity-121.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-121.png,Lung_Opacity/masks/Lung_Opacity-121.png,True,False,299,299,L,150.56,bd84d938c23095ef385e2104a3205a3c
Lung_Opacity-1210.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1210.png,Lung_Opacity/images/Lung_Opacity-1210.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1210.png,Lung_Opacity/masks/Lung_Opacity-1210.png,True,False,299,299,L,119.14,84521079c9fc24e8de1f0fb47b382c87
Lung_Opacity-1211.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1211.png,Lung_Opacity/images/Lung_Opacity-1211.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1211.png,Lung_Opacity/masks/Lung_Opacity-1211.png,True,False,299,299,L,130.41,d9336eaa0fb49ae0d9b1099d9371c27f
Lung_Opacity-1212.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1212.png,Lung_Opacity/images/Lung_Opacity-1212.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1212.png,Lung_Opacity/masks/Lung_Opacity-1212.png,True,False,299,299,L,131.89,0cb391cec07307d87f30d86e62048167
Lung_Opacity-1213.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1213.png,Lung_Opacity/images/Lung_Opacity-1213.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1213.png,Lung_Opacity/masks/Lung_Opacity-1213.png,True,False,299,299,L,144.97,e9856f539481386c639f72cb2d1b18b8
Lung_Opacity-1214.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1214.png,Lung_Opacity/images/Lung_Opacity-1214.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1214.png,Lung_Opacity/masks/Lung_Opacity-1214.png,True,False,299,299,L,100.5,b494cc5c34fd0e5d690fed54034d17f5
Lung_Opacity-1215.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1215.png,Lung_Opacity/images/Lung_Opacity-1215.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1215.png,Lung_Opacity/masks/Lung_Opacity-1215.png,True,False,299,299,L,156.39,3f7f1ba27aa51c8bc1e0b7134ca768c1
Lung_Opacity-1216.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1216.png,Lung_Opacity/images/Lung_Opacity-1216.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1216.png,Lung_Opacity/masks/Lung_Opacity-1216.png,True,False,299,299,L,115.05,5930b8f6572beca4c9f86f0eff622f20
Lung_Opacity-1217.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1217.png,Lung_Opacity/images/Lung_Opacity-1217.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1217.png,Lung_Opacity/masks/Lung_Opacity-1217.png,True,False,299,299,L,142.65,aa775dedce6d612881b8095d3d2f6751
Lung_Opacity-1218.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1218.png,Lung_Opacity/images/Lung_Opacity-1218.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1218.png,Lung_Opacity/masks/Lung_Opacity-1218.png,True,False,299,299,L,102.33,46c25fb2152eb7b2342097d70e1592e1
Lung_Opacity-1219.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1219.png,Lung_Opacity/images/Lung_Opacity-1219.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1219.png,Lung_Opacity/masks/Lung_Opacity-1219.png,True,False,299,299,L,121.5,c09d815ef7a25b344ad2708453d2bcbd
Lung_Opacity-122.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-122.png,Lung_Opacity/images/Lung_Opacity-122.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-122.png,Lung_Opacity/masks/Lung_Opacity-122.png,True,False,299,299,L,171.82,e350897776dab85f46e571da15d205d6
Lung_Opacity-1220.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1220.png,Lung_Opacity/images/Lung_Opacity-1220.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1220.png,Lung_Opacity/masks/Lung_Opacity-1220.png,True,False,299,299,L,117.54,c98a9e5d9a76ebb0e7c320094b498c83
Lung_Opacity-1221.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1221.png,Lung_Opacity/images/Lung_Opacity-1221.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1221.png,Lung_Opacity/masks/Lung_Opacity-1221.png,True,False,299,299,L,131.94,973519707d86beaad6306f7cf04dd972
Lung_Opacity-1222.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1222.png,Lung_Opacity/images/Lung_Opacity-1222.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1222.png,Lung_Opacity/masks/Lung_Opacity-1222.png,True,False,299,299,L,116.24,0d7c4f6ea4dd55a41e488a2f3a9979ee
Lung_Opacity-1223.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1223.png,Lung_Opacity/images/Lung_Opacity-1223.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1223.png,Lung_Opacity/masks/Lung_Opacity-1223.png,True,False,299,299,L,107.7,a3b73d439e43ab0f81ff5be6bb2ef656
Lung_Opacity-1224.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1224.png,Lung_Opacity/images/Lung_Opacity-1224.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1224.png,Lung_Opacity/masks/Lung_Opacity-1224.png,True,False,299,299,L,107.94,0cbcfff36ef2f38e1afd75c16a35bd4d
Lung_Opacity-1225.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1225.png,Lung_Opacity/images/Lung_Opacity-1225.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1225.png,Lung_Opacity/masks/Lung_Opacity-1225.png,True,False,299,299,L,98.68,3ea0ea2a39b35eb64422755eb6767b92
Lung_Opacity-1226.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1226.png,Lung_Opacity/images/Lung_Opacity-1226.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1226.png,Lung_Opacity/masks/Lung_Opacity-1226.png,True,False,299,299,L,122.14,a4556bec2bd63d54c208858433765830
Lung_Opacity-1227.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1227.png,Lung_Opacity/images/Lung_Opacity-1227.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1227.png,Lung_Opacity/masks/Lung_Opacity-1227.png,True,False,299,299,L,114.58,f0c573c36ff262a84baaea7e6d93cc0a
Lung_Opacity-1228.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1228.png,Lung_Opacity/images/Lung_Opacity-1228.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1228.png,Lung_Opacity/masks/Lung_Opacity-1228.png,True,False,299,299,L,124.96,d14a31a6a3a09656f7cb49244ab2fb0c
Lung_Opacity-1229.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1229.png,Lung_Opacity/images/Lung_Opacity-1229.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1229.png,Lung_Opacity/masks/Lung_Opacity-1229.png,True,False,299,299,L,151.24,cfe96cb4fd33cf9955334749c90978cd
Lung_Opacity-123.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-123.png,Lung_Opacity/images/Lung_Opacity-123.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-123.png,Lung_Opacity/masks/Lung_Opacity-123.png,True,False,299,299,L,117.3,b60a085415da4b4dad95ded957aa6c16
Lung_Opacity-1230.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1230.png,Lung_Opacity/images/Lung_Opacity-1230.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1230.png,Lung_Opacity/masks/Lung_Opacity-1230.png,True,False,299,299,L,104.63,f5360c6b3adbfb3bcbc3f422e2c26e0c
Lung_Opacity-1231.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1231.png,Lung_Opacity/images/Lung_Opacity-1231.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1231.png,Lung_Opacity/masks/Lung_Opacity-1231.png,True,False,299,299,L,118.3,1eb50d0ef04104072aaeaf336ef2f267
Lung_Opacity-1232.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1232.png,Lung_Opacity/images/Lung_Opacity-1232.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1232.png,Lung_Opacity/masks/Lung_Opacity-1232.png,True,False,299,299,L,115.59,0bb418bd30364d7cb5441375546d2bf8
Lung_Opacity-1233.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1233.png,Lung_Opacity/images/Lung_Opacity-1233.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1233.png,Lung_Opacity/masks/Lung_Opacity-1233.png,True,False,299,299,L,115.36,b508a980e95dd79e41858cab1adcf59f
Lung_Opacity-1234.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1234.png,Lung_Opacity/images/Lung_Opacity-1234.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1234.png,Lung_Opacity/masks/Lung_Opacity-1234.png,True,False,299,299,L,158.46,c4058e7b6269bc76ef41cfc5247c4128
Lung_Opacity-1235.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1235.png,Lung_Opacity/images/Lung_Opacity-1235.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1235.png,Lung_Opacity/masks/Lung_Opacity-1235.png,True,False,299,299,L,107.27,faf410254fab51bdca2e9beebd887b77
Lung_Opacity-1236.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1236.png,Lung_Opacity/images/Lung_Opacity-1236.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1236.png,Lung_Opacity/masks/Lung_Opacity-1236.png,True,False,299,299,L,126.74,4748e509fcba82afae79774a1daf463c
Lung_Opacity-1237.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1237.png,Lung_Opacity/images/Lung_Opacity-1237.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1237.png,Lung_Opacity/masks/Lung_Opacity-1237.png,True,False,299,299,L,108.03,2f8cc7c18aed13af52da4217dcbfa9b0
Lung_Opacity-1238.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1238.png,Lung_Opacity/images/Lung_Opacity-1238.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1238.png,Lung_Opacity/masks/Lung_Opacity-1238.png,True,False,299,299,L,88.23,c6e41e8f6cf70e9f71c40d9997099d64
Lung_Opacity-1239.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1239.png,Lung_Opacity/images/Lung_Opacity-1239.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1239.png,Lung_Opacity/masks/Lung_Opacity-1239.png,True,False,299,299,L,131.09,c69f1ef466720f9c217e3ee05316f597
Lung_Opacity-124.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-124.png,Lung_Opacity/images/Lung_Opacity-124.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-124.png,Lung_Opacity/masks/Lung_Opacity-124.png,True,False,299,299,L,149.25,fd34794b802c46b6ac6d4b635e7f1e77
Lung_Opacity-1240.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1240.png,Lung_Opacity/images/Lung_Opacity-1240.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1240.png,Lung_Opacity/masks/Lung_Opacity-1240.png,True,False,299,299,L,121.19,dbbcc3bf33164407b831ad1b9213b122
Lung_Opacity-1241.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1241.png,Lung_Opacity/images/Lung_Opacity-1241.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1241.png,Lung_Opacity/masks/Lung_Opacity-1241.png,True,False,299,299,L,114.91,8b4ae6222bbe5c3fdd1587313e9e02c4
Lung_Opacity-1242.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1242.png,Lung_Opacity/images/Lung_Opacity-1242.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1242.png,Lung_Opacity/masks/Lung_Opacity-1242.png,True,False,299,299,L,100.92,e06ff7cb433d23ada1366acfe9b79900
Lung_Opacity-1243.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1243.png,Lung_Opacity/images/Lung_Opacity-1243.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1243.png,Lung_Opacity/masks/Lung_Opacity-1243.png,True,False,299,299,L,141.8,9c5d241aadf75286370ee30d1b2242c9
Lung_Opacity-1244.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1244.png,Lung_Opacity/images/Lung_Opacity-1244.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1244.png,Lung_Opacity/masks/Lung_Opacity-1244.png,True,False,299,299,L,129.04,7533b4281c770e3114a042c511bbdd32
Lung_Opacity-1245.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1245.png,Lung_Opacity/images/Lung_Opacity-1245.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1245.png,Lung_Opacity/masks/Lung_Opacity-1245.png,True,False,299,299,L,108.79,0ff23a637662f517eb128b9590cfb306
Lung_Opacity-1246.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1246.png,Lung_Opacity/images/Lung_Opacity-1246.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1246.png,Lung_Opacity/masks/Lung_Opacity-1246.png,True,False,299,299,L,103.64,d145d565c4f78c245e183ea66af26491
Lung_Opacity-1247.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1247.png,Lung_Opacity/images/Lung_Opacity-1247.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1247.png,Lung_Opacity/masks/Lung_Opacity-1247.png,True,False,299,299,L,113.81,bb16360e543d084f75b4017144eea064
Lung_Opacity-1248.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1248.png,Lung_Opacity/images/Lung_Opacity-1248.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1248.png,Lung_Opacity/masks/Lung_Opacity-1248.png,True,False,299,299,L,101.16,636511dafa83ee27dc6dda0e3c823169
Lung_Opacity-1249.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1249.png,Lung_Opacity/images/Lung_Opacity-1249.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1249.png,Lung_Opacity/masks/Lung_Opacity-1249.png,True,False,299,299,L,108.41,78fc46b7a9dc3ad4803b5d7a7a01dbad
Lung_Opacity-125.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-125.png,Lung_Opacity/images/Lung_Opacity-125.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-125.png,Lung_Opacity/masks/Lung_Opacity-125.png,True,False,299,299,L,90.92,80c8da8227ed7ae97950dc3be9175591
Lung_Opacity-1250.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1250.png,Lung_Opacity/images/Lung_Opacity-1250.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1250.png,Lung_Opacity/masks/Lung_Opacity-1250.png,True,False,299,299,L,107.23,96b0a2ea1dbbf9fd54c764c966e61f5f
Lung_Opacity-1251.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1251.png,Lung_Opacity/images/Lung_Opacity-1251.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1251.png,Lung_Opacity/masks/Lung_Opacity-1251.png,True,False,299,299,L,140.4,debf2422e0b3d7899ee48d0522d3f9bf
Lung_Opacity-1252.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1252.png,Lung_Opacity/images/Lung_Opacity-1252.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1252.png,Lung_Opacity/masks/Lung_Opacity-1252.png,True,False,299,299,L,84.54,a9f94a3d52463a3b9b34bbeae9f9a299
Lung_Opacity-1253.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1253.png,Lung_Opacity/images/Lung_Opacity-1253.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1253.png,Lung_Opacity/masks/Lung_Opacity-1253.png,True,False,299,299,L,115.84,5af168d096d3cbbc4ec0ef387d13990b
Lung_Opacity-1254.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1254.png,Lung_Opacity/images/Lung_Opacity-1254.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1254.png,Lung_Opacity/masks/Lung_Opacity-1254.png,True,False,299,299,L,104.67,19f19e355190cab7623d2c4febdb7ba6
Lung_Opacity-1255.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1255.png,Lung_Opacity/images/Lung_Opacity-1255.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1255.png,Lung_Opacity/masks/Lung_Opacity-1255.png,True,False,299,299,L,110.11,c301b1213f2ff5eaba9737d21a514a64
Lung_Opacity-1256.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1256.png,Lung_Opacity/images/Lung_Opacity-1256.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1256.png,Lung_Opacity/masks/Lung_Opacity-1256.png,True,False,299,299,L,103.67,a1f806e8f0b4d20d20ae9325c5521770
Lung_Opacity-1257.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1257.png,Lung_Opacity/images/Lung_Opacity-1257.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1257.png,Lung_Opacity/masks/Lung_Opacity-1257.png,True,False,299,299,L,111.8,360d4e68d43cbc99716943e3910f155b
Lung_Opacity-1258.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1258.png,Lung_Opacity/images/Lung_Opacity-1258.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1258.png,Lung_Opacity/masks/Lung_Opacity-1258.png,True,False,299,299,L,97.65,e2a6f7acd50b3c9b650decb391e6688f
Lung_Opacity-1259.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1259.png,Lung_Opacity/images/Lung_Opacity-1259.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1259.png,Lung_Opacity/masks/Lung_Opacity-1259.png,True,False,299,299,L,91.5,c1dce79a400bc8bac8b175acbd093d58
Lung_Opacity-126.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-126.png,Lung_Opacity/images/Lung_Opacity-126.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-126.png,Lung_Opacity/masks/Lung_Opacity-126.png,True,False,299,299,L,145.44,caf00593bdda7f6a2c6def82cdd6b4e8
Lung_Opacity-1260.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1260.png,Lung_Opacity/images/Lung_Opacity-1260.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1260.png,Lung_Opacity/masks/Lung_Opacity-1260.png,True,False,299,299,L,117.66,c990a363ce7d0210620d1a86e7360a0f
Lung_Opacity-1261.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1261.png,Lung_Opacity/images/Lung_Opacity-1261.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1261.png,Lung_Opacity/masks/Lung_Opacity-1261.png,True,False,299,299,L,109.83,90232276f5016b6050c92325fb78389e
Lung_Opacity-1262.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1262.png,Lung_Opacity/images/Lung_Opacity-1262.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1262.png,Lung_Opacity/masks/Lung_Opacity-1262.png,True,False,299,299,L,89.19,c45d0c5e247871fd3760334e2f242563
Lung_Opacity-1263.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1263.png,Lung_Opacity/images/Lung_Opacity-1263.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1263.png,Lung_Opacity/masks/Lung_Opacity-1263.png,True,False,299,299,L,166.14,6e06a845aa3c51dc6d9238944dacb401
Lung_Opacity-1264.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1264.png,Lung_Opacity/images/Lung_Opacity-1264.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1264.png,Lung_Opacity/masks/Lung_Opacity-1264.png,True,False,299,299,L,79.94,9609dcf9e37d9ee1983fec9fa0821fbf
Lung_Opacity-1265.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1265.png,Lung_Opacity/images/Lung_Opacity-1265.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1265.png,Lung_Opacity/masks/Lung_Opacity-1265.png,True,False,299,299,L,105.12,846d33ebbfe9e7c6ac0d630158dee077
Lung_Opacity-1266.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1266.png,Lung_Opacity/images/Lung_Opacity-1266.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1266.png,Lung_Opacity/masks/Lung_Opacity-1266.png,True,False,299,299,L,147.05,0539cb9df3c13d7156e5317b0aa88bb2
Lung_Opacity-1267.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1267.png,Lung_Opacity/images/Lung_Opacity-1267.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1267.png,Lung_Opacity/masks/Lung_Opacity-1267.png,True,False,299,299,L,166.46,fd5a5e4598873af92d6584ea49e83e9f
Lung_Opacity-1268.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1268.png,Lung_Opacity/images/Lung_Opacity-1268.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1268.png,Lung_Opacity/masks/Lung_Opacity-1268.png,True,False,299,299,L,125.14,7a57f5a87286de36e7010aec6b67b7aa
Lung_Opacity-1269.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1269.png,Lung_Opacity/images/Lung_Opacity-1269.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1269.png,Lung_Opacity/masks/Lung_Opacity-1269.png,True,False,299,299,L,149.65,cadae9aa1b85a01d45cfcf08005e8687
Lung_Opacity-127.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-127.png,Lung_Opacity/images/Lung_Opacity-127.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-127.png,Lung_Opacity/masks/Lung_Opacity-127.png,True,False,299,299,L,122.57,9333bb8b26bb0c1799b44ccd91900b82
Lung_Opacity-1270.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1270.png,Lung_Opacity/images/Lung_Opacity-1270.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1270.png,Lung_Opacity/masks/Lung_Opacity-1270.png,True,False,299,299,L,139.3,b7c541b85aad30378963158533461e71
Lung_Opacity-1271.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1271.png,Lung_Opacity/images/Lung_Opacity-1271.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1271.png,Lung_Opacity/masks/Lung_Opacity-1271.png,True,False,299,299,L,90.19,bc37908a9f019a0d416f7471a24c2209
Lung_Opacity-1272.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1272.png,Lung_Opacity/images/Lung_Opacity-1272.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1272.png,Lung_Opacity/masks/Lung_Opacity-1272.png,True,False,299,299,L,98.2,8d9a7571215980091d4993b5208d5e95
Lung_Opacity-1273.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1273.png,Lung_Opacity/images/Lung_Opacity-1273.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1273.png,Lung_Opacity/masks/Lung_Opacity-1273.png,True,False,299,299,L,116.47,ce2b9a01b014f4ada999d190a41c076c
Lung_Opacity-1274.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1274.png,Lung_Opacity/images/Lung_Opacity-1274.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1274.png,Lung_Opacity/masks/Lung_Opacity-1274.png,True,False,299,299,L,118.65,ea44ccfd250d70023f90dca1c2471274
Lung_Opacity-1275.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1275.png,Lung_Opacity/images/Lung_Opacity-1275.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1275.png,Lung_Opacity/masks/Lung_Opacity-1275.png,True,False,299,299,L,171.04,abf30aaf00d289026d9dffc552282c67
Lung_Opacity-1276.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1276.png,Lung_Opacity/images/Lung_Opacity-1276.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1276.png,Lung_Opacity/masks/Lung_Opacity-1276.png,True,False,299,299,L,132.29,0c0fe0e772c8453fe06ff751f126ece9
Lung_Opacity-1277.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1277.png,Lung_Opacity/images/Lung_Opacity-1277.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1277.png,Lung_Opacity/masks/Lung_Opacity-1277.png,True,False,299,299,L,146.77,f61925f613b89011f4f812135236a9f1
Lung_Opacity-1278.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1278.png,Lung_Opacity/images/Lung_Opacity-1278.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1278.png,Lung_Opacity/masks/Lung_Opacity-1278.png,True,False,299,299,L,153.48,14854a1666f6e8127b784dda0556571c
Lung_Opacity-1279.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1279.png,Lung_Opacity/images/Lung_Opacity-1279.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1279.png,Lung_Opacity/masks/Lung_Opacity-1279.png,True,False,299,299,L,147.39,eed76b7dcb857cbe2f946d31e768aa41
Lung_Opacity-128.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-128.png,Lung_Opacity/images/Lung_Opacity-128.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-128.png,Lung_Opacity/masks/Lung_Opacity-128.png,True,False,299,299,L,128.02,c2f116ef8562deb7686f04f2008e76a4
Lung_Opacity-1280.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1280.png,Lung_Opacity/images/Lung_Opacity-1280.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1280.png,Lung_Opacity/masks/Lung_Opacity-1280.png,True,False,299,299,L,138.43,09757cecc2acd310a39ed9c65c8a0aa2
Lung_Opacity-1281.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1281.png,Lung_Opacity/images/Lung_Opacity-1281.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1281.png,Lung_Opacity/masks/Lung_Opacity-1281.png,True,False,299,299,L,95.92,6094b8d8822d90cd06f5358a5679851f
Lung_Opacity-1282.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1282.png,Lung_Opacity/images/Lung_Opacity-1282.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1282.png,Lung_Opacity/masks/Lung_Opacity-1282.png,True,False,299,299,L,124.01,5ffa1596d6d1b903657470d80b99bfa6
Lung_Opacity-1283.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1283.png,Lung_Opacity/images/Lung_Opacity-1283.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1283.png,Lung_Opacity/masks/Lung_Opacity-1283.png,True,False,299,299,L,149.62,1824bc2f149de68880c25a38911a6ff4
Lung_Opacity-1284.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1284.png,Lung_Opacity/images/Lung_Opacity-1284.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1284.png,Lung_Opacity/masks/Lung_Opacity-1284.png,True,False,299,299,L,113.54,bbbb98cb5f538084c3feee7f73b95be8
Lung_Opacity-1285.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1285.png,Lung_Opacity/images/Lung_Opacity-1285.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1285.png,Lung_Opacity/masks/Lung_Opacity-1285.png,True,False,299,299,L,114.2,149d51a33fcc83f7eaf9498806b42c79
Lung_Opacity-1286.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1286.png,Lung_Opacity/images/Lung_Opacity-1286.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1286.png,Lung_Opacity/masks/Lung_Opacity-1286.png,True,False,299,299,L,151.64,2f2d040b0b5f4f81e11d85283b368a7f
Lung_Opacity-1287.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1287.png,Lung_Opacity/images/Lung_Opacity-1287.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1287.png,Lung_Opacity/masks/Lung_Opacity-1287.png,True,False,299,299,L,153.24,97e9d84cd892a36e47ec73392cc1ebc6
Lung_Opacity-1288.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1288.png,Lung_Opacity/images/Lung_Opacity-1288.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1288.png,Lung_Opacity/masks/Lung_Opacity-1288.png,True,False,299,299,L,138.39,d00ee382959802993cef0b30f85e1453
Lung_Opacity-1289.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1289.png,Lung_Opacity/images/Lung_Opacity-1289.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1289.png,Lung_Opacity/masks/Lung_Opacity-1289.png,True,False,299,299,L,155.33,37385ed20c4628da8cfb6d00c18cfcf6
Lung_Opacity-129.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-129.png,Lung_Opacity/images/Lung_Opacity-129.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-129.png,Lung_Opacity/masks/Lung_Opacity-129.png,True,False,299,299,L,111.86,a2228013ebe565f6771942613e0b9ab7
Lung_Opacity-1290.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1290.png,Lung_Opacity/images/Lung_Opacity-1290.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1290.png,Lung_Opacity/masks/Lung_Opacity-1290.png,True,False,299,299,L,122.93,d70b1905c13046bf1a76f0898cda7106
Lung_Opacity-1291.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1291.png,Lung_Opacity/images/Lung_Opacity-1291.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1291.png,Lung_Opacity/masks/Lung_Opacity-1291.png,True,False,299,299,L,113.17,46668c8034051a4bd045b9747c33ed0a
Lung_Opacity-1292.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1292.png,Lung_Opacity/images/Lung_Opacity-1292.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1292.png,Lung_Opacity/masks/Lung_Opacity-1292.png,True,False,299,299,L,116.4,12193486c55e2494f5a82349063b00a0
Lung_Opacity-1293.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1293.png,Lung_Opacity/images/Lung_Opacity-1293.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1293.png,Lung_Opacity/masks/Lung_Opacity-1293.png,True,False,299,299,L,116.41,44a265e0ecb00d5f78362be8f4e344f0
Lung_Opacity-1294.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1294.png,Lung_Opacity/images/Lung_Opacity-1294.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1294.png,Lung_Opacity/masks/Lung_Opacity-1294.png,True,False,299,299,L,161.64,0ab6acb68c242ccb2ab4fef14274f636
Lung_Opacity-1295.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1295.png,Lung_Opacity/images/Lung_Opacity-1295.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1295.png,Lung_Opacity/masks/Lung_Opacity-1295.png,True,False,299,299,L,111.33,5eedbf8d47ccd0511cdff1486aa6ae21
Lung_Opacity-1296.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1296.png,Lung_Opacity/images/Lung_Opacity-1296.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1296.png,Lung_Opacity/masks/Lung_Opacity-1296.png,True,False,299,299,L,101.36,b2da51376d1564b913eebc399a17ed08
Lung_Opacity-1297.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1297.png,Lung_Opacity/images/Lung_Opacity-1297.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1297.png,Lung_Opacity/masks/Lung_Opacity-1297.png,True,False,299,299,L,110.16,d1ebfecaeb155dcf588a1ea2a875cefa
Lung_Opacity-1298.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1298.png,Lung_Opacity/images/Lung_Opacity-1298.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1298.png,Lung_Opacity/masks/Lung_Opacity-1298.png,True,False,299,299,L,123.77,47d996ab8e6689eda2adf6c01f9a190b
Lung_Opacity-1299.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1299.png,Lung_Opacity/images/Lung_Opacity-1299.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1299.png,Lung_Opacity/masks/Lung_Opacity-1299.png,True,False,299,299,L,114.02,75d4b35c0f48d4db9be2bbfb46633fad
Lung_Opacity-13.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-13.png,Lung_Opacity/images/Lung_Opacity-13.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-13.png,Lung_Opacity/masks/Lung_Opacity-13.png,True,False,299,299,L,113.46,db697f31e97a6919cc04baf365bf2947
Lung_Opacity-130.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-130.png,Lung_Opacity/images/Lung_Opacity-130.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-130.png,Lung_Opacity/masks/Lung_Opacity-130.png,True,False,299,299,L,94.46,23493e0c4ace773dcdef35bb2e96ddc6
Lung_Opacity-1300.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1300.png,Lung_Opacity/images/Lung_Opacity-1300.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1300.png,Lung_Opacity/masks/Lung_Opacity-1300.png,True,False,299,299,L,171.34,7213aa3099a2f850383ce885057c3532
Lung_Opacity-1301.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1301.png,Lung_Opacity/images/Lung_Opacity-1301.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1301.png,Lung_Opacity/masks/Lung_Opacity-1301.png,True,False,299,299,L,113.53,452ff682f2e784290cfb8aae547fa58a
Lung_Opacity-1302.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1302.png,Lung_Opacity/images/Lung_Opacity-1302.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1302.png,Lung_Opacity/masks/Lung_Opacity-1302.png,True,False,299,299,L,161.36,39d7e4453b3df6ff045ebe04e278478a
Lung_Opacity-1303.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1303.png,Lung_Opacity/images/Lung_Opacity-1303.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1303.png,Lung_Opacity/masks/Lung_Opacity-1303.png,True,False,299,299,L,118.1,da8b7523181e3afb24275866bdf22ab5
Lung_Opacity-1304.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1304.png,Lung_Opacity/images/Lung_Opacity-1304.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1304.png,Lung_Opacity/masks/Lung_Opacity-1304.png,True,False,299,299,L,154.4,102eda52ad236878a5e98fe76b803149
Lung_Opacity-1305.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1305.png,Lung_Opacity/images/Lung_Opacity-1305.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1305.png,Lung_Opacity/masks/Lung_Opacity-1305.png,True,False,299,299,L,159.04,8d5102448713715bd52657c4806396c5
Lung_Opacity-1306.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1306.png,Lung_Opacity/images/Lung_Opacity-1306.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1306.png,Lung_Opacity/masks/Lung_Opacity-1306.png,True,False,299,299,L,116.38,b5abc451b22c1fbedd2b4788ee94a99e
Lung_Opacity-1307.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1307.png,Lung_Opacity/images/Lung_Opacity-1307.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1307.png,Lung_Opacity/masks/Lung_Opacity-1307.png,True,False,299,299,L,161.25,8711fdb007a2dd90294346809d3dfb05
Lung_Opacity-1308.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1308.png,Lung_Opacity/images/Lung_Opacity-1308.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1308.png,Lung_Opacity/masks/Lung_Opacity-1308.png,True,False,299,299,L,126.75,927a5831904beaa4735932601c9a6c6f
Lung_Opacity-1309.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1309.png,Lung_Opacity/images/Lung_Opacity-1309.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1309.png,Lung_Opacity/masks/Lung_Opacity-1309.png,True,False,299,299,L,155.22,66ded4d3b723dacb31865577a17e2ae3
Lung_Opacity-131.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-131.png,Lung_Opacity/images/Lung_Opacity-131.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-131.png,Lung_Opacity/masks/Lung_Opacity-131.png,True,False,299,299,L,149.5,6756fbc3a635bfe640a4516be7576f15
Lung_Opacity-1310.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1310.png,Lung_Opacity/images/Lung_Opacity-1310.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1310.png,Lung_Opacity/masks/Lung_Opacity-1310.png,True,False,299,299,L,116.92,85837b0b63fb86e73e5bbb3b47c305ca
Lung_Opacity-1311.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1311.png,Lung_Opacity/images/Lung_Opacity-1311.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1311.png,Lung_Opacity/masks/Lung_Opacity-1311.png,True,False,299,299,L,149.22,cb47d52bc9f44d8e7927c85836597bb1
Lung_Opacity-1312.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1312.png,Lung_Opacity/images/Lung_Opacity-1312.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1312.png,Lung_Opacity/masks/Lung_Opacity-1312.png,True,False,299,299,L,107.79,0ca432c644cd605bc0f8da5688de0608
Lung_Opacity-1313.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1313.png,Lung_Opacity/images/Lung_Opacity-1313.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1313.png,Lung_Opacity/masks/Lung_Opacity-1313.png,True,False,299,299,L,134.4,b372c79d25d1c70f85f78a4a3a48ff17
Lung_Opacity-1314.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1314.png,Lung_Opacity/images/Lung_Opacity-1314.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1314.png,Lung_Opacity/masks/Lung_Opacity-1314.png,True,False,299,299,L,115.07,786ea8b772b940603b155ac2e24588d7
Lung_Opacity-1315.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1315.png,Lung_Opacity/images/Lung_Opacity-1315.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1315.png,Lung_Opacity/masks/Lung_Opacity-1315.png,True,False,299,299,L,117.89,bb44fb1ad6fcd685f5cf0060e3e9ea7e
Lung_Opacity-1316.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1316.png,Lung_Opacity/images/Lung_Opacity-1316.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1316.png,Lung_Opacity/masks/Lung_Opacity-1316.png,True,False,299,299,L,106.9,d9e418d73cf4a2a031f60191bccd6ad0
Lung_Opacity-1317.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1317.png,Lung_Opacity/images/Lung_Opacity-1317.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1317.png,Lung_Opacity/masks/Lung_Opacity-1317.png,True,False,299,299,L,130.9,c4f76a812ad4a02ae5fe8599877bdf6b
Lung_Opacity-1318.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1318.png,Lung_Opacity/images/Lung_Opacity-1318.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1318.png,Lung_Opacity/masks/Lung_Opacity-1318.png,True,False,299,299,L,163.88,4eb6e4d54d95d2ccbbda4ecdce5965e6
Lung_Opacity-1319.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1319.png,Lung_Opacity/images/Lung_Opacity-1319.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1319.png,Lung_Opacity/masks/Lung_Opacity-1319.png,True,False,299,299,L,166.44,bb6af16a7f505f7c07823cfc1390a2a7
Lung_Opacity-132.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-132.png,Lung_Opacity/images/Lung_Opacity-132.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-132.png,Lung_Opacity/masks/Lung_Opacity-132.png,True,False,299,299,L,115.84,3fc48953a0a13866dd9c724994e18401
Lung_Opacity-1320.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1320.png,Lung_Opacity/images/Lung_Opacity-1320.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1320.png,Lung_Opacity/masks/Lung_Opacity-1320.png,True,False,299,299,L,120.68,1fcd5e910a6033df2b55caec38b6ecec
Lung_Opacity-1321.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1321.png,Lung_Opacity/images/Lung_Opacity-1321.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1321.png,Lung_Opacity/masks/Lung_Opacity-1321.png,True,False,299,299,L,101.53,e029874ba424914444fc7cc9eb036cff
Lung_Opacity-1322.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1322.png,Lung_Opacity/images/Lung_Opacity-1322.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1322.png,Lung_Opacity/masks/Lung_Opacity-1322.png,True,False,299,299,L,114.68,9886086bcc66c12f7a6bfece5a8e8877
Lung_Opacity-1323.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1323.png,Lung_Opacity/images/Lung_Opacity-1323.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1323.png,Lung_Opacity/masks/Lung_Opacity-1323.png,True,False,299,299,L,169.41,913fac8bbed6362f9129bc9916da6a52
Lung_Opacity-1324.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1324.png,Lung_Opacity/images/Lung_Opacity-1324.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1324.png,Lung_Opacity/masks/Lung_Opacity-1324.png,True,False,299,299,L,100.74,6e1bafc8bb475985cdddfd34c429e6b2
Lung_Opacity-1325.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1325.png,Lung_Opacity/images/Lung_Opacity-1325.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1325.png,Lung_Opacity/masks/Lung_Opacity-1325.png,True,False,299,299,L,130.63,a530e5789933fb2ac02deaf3224efe97
Lung_Opacity-1326.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1326.png,Lung_Opacity/images/Lung_Opacity-1326.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1326.png,Lung_Opacity/masks/Lung_Opacity-1326.png,True,False,299,299,L,170.87,5ada138fdbd315419c686847df9af460
Lung_Opacity-1327.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1327.png,Lung_Opacity/images/Lung_Opacity-1327.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1327.png,Lung_Opacity/masks/Lung_Opacity-1327.png,True,False,299,299,L,111.13,0eb98298afb079338da6af0e6043d085
Lung_Opacity-1328.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1328.png,Lung_Opacity/images/Lung_Opacity-1328.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1328.png,Lung_Opacity/masks/Lung_Opacity-1328.png,True,False,299,299,L,160.33,d1f37e01cd91a1bdf9d4de44e4404a49
Lung_Opacity-1329.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1329.png,Lung_Opacity/images/Lung_Opacity-1329.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1329.png,Lung_Opacity/masks/Lung_Opacity-1329.png,True,False,299,299,L,98.74,a4ff0f5f817a765c92b8f631b2e674b8
Lung_Opacity-133.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-133.png,Lung_Opacity/images/Lung_Opacity-133.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-133.png,Lung_Opacity/masks/Lung_Opacity-133.png,True,False,299,299,L,126.02,c1ee9106852a3ef1780f3ea183e625fd
Lung_Opacity-1330.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1330.png,Lung_Opacity/images/Lung_Opacity-1330.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1330.png,Lung_Opacity/masks/Lung_Opacity-1330.png,True,False,299,299,L,121.1,d6a76886376d51c33b5ebea256c1dc9d
Lung_Opacity-1331.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1331.png,Lung_Opacity/images/Lung_Opacity-1331.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1331.png,Lung_Opacity/masks/Lung_Opacity-1331.png,True,False,299,299,L,152.09,dcf18f9d5697bbc21010659069983272
Lung_Opacity-1332.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1332.png,Lung_Opacity/images/Lung_Opacity-1332.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1332.png,Lung_Opacity/masks/Lung_Opacity-1332.png,True,False,299,299,L,130.39,d31e1cf79af33af00f979e3f4f9ea4f9
Lung_Opacity-1333.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1333.png,Lung_Opacity/images/Lung_Opacity-1333.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1333.png,Lung_Opacity/masks/Lung_Opacity-1333.png,True,False,299,299,L,114.54,5d2a3777a1029e773ef89fc8cc9e74fc
Lung_Opacity-1334.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1334.png,Lung_Opacity/images/Lung_Opacity-1334.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1334.png,Lung_Opacity/masks/Lung_Opacity-1334.png,True,False,299,299,L,108.51,d1345a1539eb8335900bc571460f4065
Lung_Opacity-1335.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1335.png,Lung_Opacity/images/Lung_Opacity-1335.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1335.png,Lung_Opacity/masks/Lung_Opacity-1335.png,True,False,299,299,L,128.11,1ee079bb6be25da995f02083959a60b8
Lung_Opacity-1336.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1336.png,Lung_Opacity/images/Lung_Opacity-1336.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1336.png,Lung_Opacity/masks/Lung_Opacity-1336.png,True,False,299,299,L,106.8,957094726d93238d8dc0a0f30a9b111f
Lung_Opacity-1337.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1337.png,Lung_Opacity/images/Lung_Opacity-1337.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1337.png,Lung_Opacity/masks/Lung_Opacity-1337.png,True,False,299,299,L,127.27,b5000aca6c2bff04cdd07d6ba5f3230b
Lung_Opacity-1338.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1338.png,Lung_Opacity/images/Lung_Opacity-1338.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1338.png,Lung_Opacity/masks/Lung_Opacity-1338.png,True,False,299,299,L,84.54,3678cb6f0ed91a548d4f3d100a0df272
Lung_Opacity-1339.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1339.png,Lung_Opacity/images/Lung_Opacity-1339.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1339.png,Lung_Opacity/masks/Lung_Opacity-1339.png,True,False,299,299,L,113.24,5a79336e895a3bc36aaecb98490cb96e
Lung_Opacity-134.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-134.png,Lung_Opacity/images/Lung_Opacity-134.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-134.png,Lung_Opacity/masks/Lung_Opacity-134.png,True,False,299,299,L,118.53,591f17893764df752e8527c4e48e52a3
Lung_Opacity-1340.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1340.png,Lung_Opacity/images/Lung_Opacity-1340.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1340.png,Lung_Opacity/masks/Lung_Opacity-1340.png,True,False,299,299,L,110.86,90cd93542d8593efb650b41f1b8574e6
Lung_Opacity-1341.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1341.png,Lung_Opacity/images/Lung_Opacity-1341.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1341.png,Lung_Opacity/masks/Lung_Opacity-1341.png,True,False,299,299,L,113.13,7839fa1de527692b7e014bd1562349f9
Lung_Opacity-1342.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1342.png,Lung_Opacity/images/Lung_Opacity-1342.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1342.png,Lung_Opacity/masks/Lung_Opacity-1342.png,True,False,299,299,L,126.4,d22c6a2f5f4214ddb27f9c2e046416b6
Lung_Opacity-1343.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1343.png,Lung_Opacity/images/Lung_Opacity-1343.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1343.png,Lung_Opacity/masks/Lung_Opacity-1343.png,True,False,299,299,L,116.33,52cb3394840f0205647d04ed7dbce68c
Lung_Opacity-1344.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1344.png,Lung_Opacity/images/Lung_Opacity-1344.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1344.png,Lung_Opacity/masks/Lung_Opacity-1344.png,True,False,299,299,L,96.13,44f5d3bc3f6d5bb3db2ce8a58d24a875
Lung_Opacity-1345.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1345.png,Lung_Opacity/images/Lung_Opacity-1345.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1345.png,Lung_Opacity/masks/Lung_Opacity-1345.png,True,False,299,299,L,152.73,9ca60b032d0ebdfd7ba4eeac211015cd
Lung_Opacity-1346.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1346.png,Lung_Opacity/images/Lung_Opacity-1346.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1346.png,Lung_Opacity/masks/Lung_Opacity-1346.png,True,False,299,299,L,124.67,3dfb782ea468c488744c33d6637e33ac
Lung_Opacity-1347.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1347.png,Lung_Opacity/images/Lung_Opacity-1347.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1347.png,Lung_Opacity/masks/Lung_Opacity-1347.png,True,False,299,299,L,155.52,ea0175a8202666f26546b7946f587cd2
Lung_Opacity-1348.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1348.png,Lung_Opacity/images/Lung_Opacity-1348.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1348.png,Lung_Opacity/masks/Lung_Opacity-1348.png,True,False,299,299,L,96.68,4e9de4aa77563545eb0c912d0a01c8fe
Lung_Opacity-1349.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1349.png,Lung_Opacity/images/Lung_Opacity-1349.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1349.png,Lung_Opacity/masks/Lung_Opacity-1349.png,True,False,299,299,L,179.28,b9faaa1a5671e0b3302565181a47e783
Lung_Opacity-135.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-135.png,Lung_Opacity/images/Lung_Opacity-135.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-135.png,Lung_Opacity/masks/Lung_Opacity-135.png,True,False,299,299,L,116.96,dc0450050ab8d5fc21d5bc16fd154085
Lung_Opacity-1350.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1350.png,Lung_Opacity/images/Lung_Opacity-1350.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1350.png,Lung_Opacity/masks/Lung_Opacity-1350.png,True,False,299,299,L,114.7,bdc73c94ef812cfa91ecc6b8c57f74d7
Lung_Opacity-1351.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1351.png,Lung_Opacity/images/Lung_Opacity-1351.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1351.png,Lung_Opacity/masks/Lung_Opacity-1351.png,True,False,299,299,L,123.52,7eea0c0c77ce27a6ddf653d3adbc994e
Lung_Opacity-1352.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1352.png,Lung_Opacity/images/Lung_Opacity-1352.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1352.png,Lung_Opacity/masks/Lung_Opacity-1352.png,True,False,299,299,L,109.72,2cf4f4fd91fefa61cb8c7562ee4ca95a
Lung_Opacity-1353.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1353.png,Lung_Opacity/images/Lung_Opacity-1353.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1353.png,Lung_Opacity/masks/Lung_Opacity-1353.png,True,False,299,299,L,90.66,98d982f9b57454f39bdef179bc92d043
Lung_Opacity-1354.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1354.png,Lung_Opacity/images/Lung_Opacity-1354.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1354.png,Lung_Opacity/masks/Lung_Opacity-1354.png,True,False,299,299,L,118.62,2fc1eed8bf94c39773afdf88dd49f03e
Lung_Opacity-1355.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1355.png,Lung_Opacity/images/Lung_Opacity-1355.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1355.png,Lung_Opacity/masks/Lung_Opacity-1355.png,True,False,299,299,L,108.22,27c5de52c4824168926c0d8e2bc50239
Lung_Opacity-1356.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1356.png,Lung_Opacity/images/Lung_Opacity-1356.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1356.png,Lung_Opacity/masks/Lung_Opacity-1356.png,True,False,299,299,L,123.26,360fbbb701d60f2498368c181895a7a7
Lung_Opacity-1357.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1357.png,Lung_Opacity/images/Lung_Opacity-1357.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1357.png,Lung_Opacity/masks/Lung_Opacity-1357.png,True,False,299,299,L,135.14,e59921e113d50dcea774f616fc89e694
Lung_Opacity-1358.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1358.png,Lung_Opacity/images/Lung_Opacity-1358.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1358.png,Lung_Opacity/masks/Lung_Opacity-1358.png,True,False,299,299,L,116.69,553d00ab1dbdd4f7b76ac3d0fb8a8324
Lung_Opacity-1359.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1359.png,Lung_Opacity/images/Lung_Opacity-1359.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1359.png,Lung_Opacity/masks/Lung_Opacity-1359.png,True,False,299,299,L,144.1,9b3b1dcac4541baff2d5e78b4db26cff
Lung_Opacity-136.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-136.png,Lung_Opacity/images/Lung_Opacity-136.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-136.png,Lung_Opacity/masks/Lung_Opacity-136.png,True,False,299,299,L,119.55,524dff86b2a918359c742024b305e733
Lung_Opacity-1360.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1360.png,Lung_Opacity/images/Lung_Opacity-1360.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1360.png,Lung_Opacity/masks/Lung_Opacity-1360.png,True,False,299,299,L,167.82,42d552db141729f92a1dc5fb0791b4b3
Lung_Opacity-1361.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1361.png,Lung_Opacity/images/Lung_Opacity-1361.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1361.png,Lung_Opacity/masks/Lung_Opacity-1361.png,True,False,299,299,L,151.11,d9e18197109684788f8138e7a3029e27
Lung_Opacity-1362.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1362.png,Lung_Opacity/images/Lung_Opacity-1362.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1362.png,Lung_Opacity/masks/Lung_Opacity-1362.png,True,False,299,299,L,70.87,e8109ddc4e7c52cae26e046537910555
Lung_Opacity-1363.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1363.png,Lung_Opacity/images/Lung_Opacity-1363.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1363.png,Lung_Opacity/masks/Lung_Opacity-1363.png,True,False,299,299,L,99.4,af449b78b19c2fa1e997bc8894532e69
Lung_Opacity-1364.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1364.png,Lung_Opacity/images/Lung_Opacity-1364.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1364.png,Lung_Opacity/masks/Lung_Opacity-1364.png,True,False,299,299,L,108.71,e01e27aae9f869e8f6b31d62d8656690
Lung_Opacity-1365.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1365.png,Lung_Opacity/images/Lung_Opacity-1365.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1365.png,Lung_Opacity/masks/Lung_Opacity-1365.png,True,False,299,299,L,119.5,460b9c801f0cc62b3c3072ed2758f090
Lung_Opacity-1366.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1366.png,Lung_Opacity/images/Lung_Opacity-1366.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1366.png,Lung_Opacity/masks/Lung_Opacity-1366.png,True,False,299,299,L,102.65,6682ce1131e5c4cf79f6b3b39130f08d
Lung_Opacity-1367.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1367.png,Lung_Opacity/images/Lung_Opacity-1367.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1367.png,Lung_Opacity/masks/Lung_Opacity-1367.png,True,False,299,299,L,104.19,cd5768997683d051d366fd28a2fca474
Lung_Opacity-1368.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1368.png,Lung_Opacity/images/Lung_Opacity-1368.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1368.png,Lung_Opacity/masks/Lung_Opacity-1368.png,True,False,299,299,L,98.36,49e7d660b7eccfa985bc619f8953e7bf
Lung_Opacity-1369.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1369.png,Lung_Opacity/images/Lung_Opacity-1369.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1369.png,Lung_Opacity/masks/Lung_Opacity-1369.png,True,False,299,299,L,95.7,f17f8af2755a456842453ed33ada2719
Lung_Opacity-137.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-137.png,Lung_Opacity/images/Lung_Opacity-137.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-137.png,Lung_Opacity/masks/Lung_Opacity-137.png,True,False,299,299,L,112.56,e57cee36badc49b17262cf710d0bba41
Lung_Opacity-1370.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1370.png,Lung_Opacity/images/Lung_Opacity-1370.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1370.png,Lung_Opacity/masks/Lung_Opacity-1370.png,True,False,299,299,L,114.93,fd0d7aab6e3ed3edb529418c95ff5dff
Lung_Opacity-1371.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1371.png,Lung_Opacity/images/Lung_Opacity-1371.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1371.png,Lung_Opacity/masks/Lung_Opacity-1371.png,True,False,299,299,L,102.97,f5420ff54164e5d1c025eb9b4aba119f
Lung_Opacity-1372.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1372.png,Lung_Opacity/images/Lung_Opacity-1372.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1372.png,Lung_Opacity/masks/Lung_Opacity-1372.png,True,False,299,299,L,108.92,bba9e5543fbb070ee118605d5710fbb5
Lung_Opacity-1373.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1373.png,Lung_Opacity/images/Lung_Opacity-1373.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1373.png,Lung_Opacity/masks/Lung_Opacity-1373.png,True,False,299,299,L,131.07,9f2b974228e5ef1fe27ea7ac242d0e74
Lung_Opacity-1374.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1374.png,Lung_Opacity/images/Lung_Opacity-1374.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1374.png,Lung_Opacity/masks/Lung_Opacity-1374.png,True,False,299,299,L,167.6,ed9fdbb35e29197c10750638383c389b
Lung_Opacity-1375.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1375.png,Lung_Opacity/images/Lung_Opacity-1375.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1375.png,Lung_Opacity/masks/Lung_Opacity-1375.png,True,False,299,299,L,106.78,68dd13fb8e6908b6c893fc8f4a7fed1f
Lung_Opacity-1376.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1376.png,Lung_Opacity/images/Lung_Opacity-1376.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1376.png,Lung_Opacity/masks/Lung_Opacity-1376.png,True,False,299,299,L,132.77,a316a30a8b35f105b9b334f74c9eae24
Lung_Opacity-1377.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1377.png,Lung_Opacity/images/Lung_Opacity-1377.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1377.png,Lung_Opacity/masks/Lung_Opacity-1377.png,True,False,299,299,L,131.6,81a1302413971896ec9afe2d21a0494b
Lung_Opacity-1378.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1378.png,Lung_Opacity/images/Lung_Opacity-1378.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1378.png,Lung_Opacity/masks/Lung_Opacity-1378.png,True,False,299,299,L,109.76,bf5ae0b7a55b8ec8a9ce7c922ca4b9be
Lung_Opacity-1379.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1379.png,Lung_Opacity/images/Lung_Opacity-1379.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1379.png,Lung_Opacity/masks/Lung_Opacity-1379.png,True,False,299,299,L,135.05,55497597870ff12cbfa28e8b7869c3c5
Lung_Opacity-138.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-138.png,Lung_Opacity/images/Lung_Opacity-138.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-138.png,Lung_Opacity/masks/Lung_Opacity-138.png,True,False,299,299,L,100.0,162c9181daf5b2f5eb03fc33608e75ff
Lung_Opacity-1380.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1380.png,Lung_Opacity/images/Lung_Opacity-1380.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1380.png,Lung_Opacity/masks/Lung_Opacity-1380.png,True,False,299,299,L,178.46,5a9be9be84df14a0b75e2656e2101667
Lung_Opacity-1381.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1381.png,Lung_Opacity/images/Lung_Opacity-1381.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1381.png,Lung_Opacity/masks/Lung_Opacity-1381.png,True,False,299,299,L,157.4,434a2f941875999522932212fc965af1
Lung_Opacity-1382.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1382.png,Lung_Opacity/images/Lung_Opacity-1382.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1382.png,Lung_Opacity/masks/Lung_Opacity-1382.png,True,False,299,299,L,174.78,40b0b15faea0b8eb6eedaa28a96f54ea
Lung_Opacity-1383.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1383.png,Lung_Opacity/images/Lung_Opacity-1383.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1383.png,Lung_Opacity/masks/Lung_Opacity-1383.png,True,False,299,299,L,175.95,b56b6180d542c1fd1a565d3c96597d6c
Lung_Opacity-1384.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1384.png,Lung_Opacity/images/Lung_Opacity-1384.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1384.png,Lung_Opacity/masks/Lung_Opacity-1384.png,True,False,299,299,L,109.64,93484e2a9a4636238724e5498e52faae
Lung_Opacity-1385.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1385.png,Lung_Opacity/images/Lung_Opacity-1385.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1385.png,Lung_Opacity/masks/Lung_Opacity-1385.png,True,False,299,299,L,100.2,a9580d9e2e8f2f0064f669feaa61fbd7
Lung_Opacity-1386.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1386.png,Lung_Opacity/images/Lung_Opacity-1386.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1386.png,Lung_Opacity/masks/Lung_Opacity-1386.png,True,False,299,299,L,121.37,32f119d47dfd8860149b177e0737a557
Lung_Opacity-1387.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1387.png,Lung_Opacity/images/Lung_Opacity-1387.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1387.png,Lung_Opacity/masks/Lung_Opacity-1387.png,True,False,299,299,L,106.03,78568fb54f0ec31ad9494985a7976c28
Lung_Opacity-1388.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1388.png,Lung_Opacity/images/Lung_Opacity-1388.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1388.png,Lung_Opacity/masks/Lung_Opacity-1388.png,True,False,299,299,L,161.18,9f7ab12e027749fd2efd5b5ccf0f53d7
Lung_Opacity-1389.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1389.png,Lung_Opacity/images/Lung_Opacity-1389.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1389.png,Lung_Opacity/masks/Lung_Opacity-1389.png,True,False,299,299,L,147.97,8170a33ffb4acb796af7daafcdb0fa09
Lung_Opacity-139.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-139.png,Lung_Opacity/images/Lung_Opacity-139.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-139.png,Lung_Opacity/masks/Lung_Opacity-139.png,True,False,299,299,L,118.57,cd98116c60d39009d4b8fefc3ea632b1
Lung_Opacity-1390.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1390.png,Lung_Opacity/images/Lung_Opacity-1390.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1390.png,Lung_Opacity/masks/Lung_Opacity-1390.png,True,False,299,299,L,112.31,ac1f0e648d7a0cfa55ae8d9e93b5344a
Lung_Opacity-1391.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1391.png,Lung_Opacity/images/Lung_Opacity-1391.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1391.png,Lung_Opacity/masks/Lung_Opacity-1391.png,True,False,299,299,L,148.23,3d4dd413e2c6b3aa04b5ffe7d594959c
Lung_Opacity-1392.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1392.png,Lung_Opacity/images/Lung_Opacity-1392.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1392.png,Lung_Opacity/masks/Lung_Opacity-1392.png,True,False,299,299,L,161.58,6262790f32352eb9effbec891767526c
Lung_Opacity-1393.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1393.png,Lung_Opacity/images/Lung_Opacity-1393.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1393.png,Lung_Opacity/masks/Lung_Opacity-1393.png,True,False,299,299,L,118.53,f2312c7180321dbd0df604c0f2825fc0
Lung_Opacity-1394.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1394.png,Lung_Opacity/images/Lung_Opacity-1394.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1394.png,Lung_Opacity/masks/Lung_Opacity-1394.png,True,False,299,299,L,121.82,dc322b348f4a1cc20a16058f1638de12
Lung_Opacity-1395.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1395.png,Lung_Opacity/images/Lung_Opacity-1395.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1395.png,Lung_Opacity/masks/Lung_Opacity-1395.png,True,False,299,299,L,189.35,d0eac658c372b5c352e9ca5a88b06ecd
Lung_Opacity-1396.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1396.png,Lung_Opacity/images/Lung_Opacity-1396.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1396.png,Lung_Opacity/masks/Lung_Opacity-1396.png,True,False,299,299,L,116.93,0ee7d534a7c7e6d356c159164d6de04f
Lung_Opacity-1397.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1397.png,Lung_Opacity/images/Lung_Opacity-1397.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1397.png,Lung_Opacity/masks/Lung_Opacity-1397.png,True,False,299,299,L,106.32,a92fb39439bb15f3346a72a1764d9bb3
Lung_Opacity-1398.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1398.png,Lung_Opacity/images/Lung_Opacity-1398.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1398.png,Lung_Opacity/masks/Lung_Opacity-1398.png,True,False,299,299,L,131.36,7bb56489aaeb20ce6f06f334eb17d8d4
Lung_Opacity-1399.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1399.png,Lung_Opacity/images/Lung_Opacity-1399.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1399.png,Lung_Opacity/masks/Lung_Opacity-1399.png,True,False,299,299,L,128.77,458ae34720b27d8c89a6ed2e1827d677
Lung_Opacity-14.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-14.png,Lung_Opacity/images/Lung_Opacity-14.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-14.png,Lung_Opacity/masks/Lung_Opacity-14.png,True,False,299,299,L,121.53,6aa2426124b75f1a7890a38521658498
Lung_Opacity-140.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-140.png,Lung_Opacity/images/Lung_Opacity-140.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-140.png,Lung_Opacity/masks/Lung_Opacity-140.png,True,False,299,299,L,110.23,99b91f8e3fdb8f5fcd43ddcdadf47aa2
Lung_Opacity-1400.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1400.png,Lung_Opacity/images/Lung_Opacity-1400.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1400.png,Lung_Opacity/masks/Lung_Opacity-1400.png,True,False,299,299,L,96.2,2fb0fcd7fa377e0770fd900d58cb4860
Lung_Opacity-1401.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1401.png,Lung_Opacity/images/Lung_Opacity-1401.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1401.png,Lung_Opacity/masks/Lung_Opacity-1401.png,True,False,299,299,L,114.85,5c22092656db1a0b6193a27a78f46245
Lung_Opacity-1402.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1402.png,Lung_Opacity/images/Lung_Opacity-1402.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1402.png,Lung_Opacity/masks/Lung_Opacity-1402.png,True,False,299,299,L,157.96,999bc9a6d940b09007d87ccfeafa0514
Lung_Opacity-1403.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1403.png,Lung_Opacity/images/Lung_Opacity-1403.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1403.png,Lung_Opacity/masks/Lung_Opacity-1403.png,True,False,299,299,L,100.93,c683d7e7bc0ff3e0aceadceaf8b5ffb5
Lung_Opacity-1404.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1404.png,Lung_Opacity/images/Lung_Opacity-1404.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1404.png,Lung_Opacity/masks/Lung_Opacity-1404.png,True,False,299,299,L,128.7,78be3d4d572c95784b187ddc52634fc0
Lung_Opacity-1405.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1405.png,Lung_Opacity/images/Lung_Opacity-1405.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1405.png,Lung_Opacity/masks/Lung_Opacity-1405.png,True,False,299,299,L,140.34,71a3365a1ce4fc72f5c53f0bf7ef90a3
Lung_Opacity-1406.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1406.png,Lung_Opacity/images/Lung_Opacity-1406.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1406.png,Lung_Opacity/masks/Lung_Opacity-1406.png,True,False,299,299,L,163.14,7b5386fe52a975735a9670c515584628
Lung_Opacity-1407.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1407.png,Lung_Opacity/images/Lung_Opacity-1407.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1407.png,Lung_Opacity/masks/Lung_Opacity-1407.png,True,False,299,299,L,102.97,87be06b293826fdd8d56fbeb033c6fcd
Lung_Opacity-1408.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1408.png,Lung_Opacity/images/Lung_Opacity-1408.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1408.png,Lung_Opacity/masks/Lung_Opacity-1408.png,True,False,299,299,L,167.54,9030976750186c2f02c0f94966a40f07
Lung_Opacity-1409.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1409.png,Lung_Opacity/images/Lung_Opacity-1409.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1409.png,Lung_Opacity/masks/Lung_Opacity-1409.png,True,False,299,299,L,112.6,00d22b5bcd570be57b9d5bdc2499e3b0
Lung_Opacity-141.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-141.png,Lung_Opacity/images/Lung_Opacity-141.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-141.png,Lung_Opacity/masks/Lung_Opacity-141.png,True,False,299,299,L,120.8,9225d65eea065d9a2021373d1294dc02
Lung_Opacity-1410.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1410.png,Lung_Opacity/images/Lung_Opacity-1410.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1410.png,Lung_Opacity/masks/Lung_Opacity-1410.png,True,False,299,299,L,153.53,f271204e71baf82eba7f81374585ba93
Lung_Opacity-1411.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1411.png,Lung_Opacity/images/Lung_Opacity-1411.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1411.png,Lung_Opacity/masks/Lung_Opacity-1411.png,True,False,299,299,L,117.84,a06917c8d0377a35d2f0a091896f306e
Lung_Opacity-1412.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1412.png,Lung_Opacity/images/Lung_Opacity-1412.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1412.png,Lung_Opacity/masks/Lung_Opacity-1412.png,True,False,299,299,L,124.2,87133886244ee316b5a5402915b2ce4d
Lung_Opacity-1413.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1413.png,Lung_Opacity/images/Lung_Opacity-1413.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1413.png,Lung_Opacity/masks/Lung_Opacity-1413.png,True,False,299,299,L,112.94,28ef13505b4beda1d7fa9f763cedce6d
Lung_Opacity-1414.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1414.png,Lung_Opacity/images/Lung_Opacity-1414.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1414.png,Lung_Opacity/masks/Lung_Opacity-1414.png,True,False,299,299,L,116.26,79fed740ccdaa1d4416dcfb286a7ea0f
Lung_Opacity-1415.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1415.png,Lung_Opacity/images/Lung_Opacity-1415.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1415.png,Lung_Opacity/masks/Lung_Opacity-1415.png,True,False,299,299,L,131.04,e6511ff5f8462622eb46ad928a8e9e91
Lung_Opacity-1416.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1416.png,Lung_Opacity/images/Lung_Opacity-1416.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1416.png,Lung_Opacity/masks/Lung_Opacity-1416.png,True,False,299,299,L,160.96,5a6897f744ee96e6c3ba9ae79c9fc790
Lung_Opacity-1417.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1417.png,Lung_Opacity/images/Lung_Opacity-1417.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1417.png,Lung_Opacity/masks/Lung_Opacity-1417.png,True,False,299,299,L,120.86,60447dc4c6901c745fba479530afdb94
Lung_Opacity-1418.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1418.png,Lung_Opacity/images/Lung_Opacity-1418.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1418.png,Lung_Opacity/masks/Lung_Opacity-1418.png,True,False,299,299,L,155.8,805733b80f102b89df44d0b110d220a3
Lung_Opacity-1419.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1419.png,Lung_Opacity/images/Lung_Opacity-1419.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1419.png,Lung_Opacity/masks/Lung_Opacity-1419.png,True,False,299,299,L,108.7,3364f52759d23deca77ed596c8ff5bc3
Lung_Opacity-142.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-142.png,Lung_Opacity/images/Lung_Opacity-142.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-142.png,Lung_Opacity/masks/Lung_Opacity-142.png,True,False,299,299,L,127.19,fe7770f479cb3a3d33b5e72f57f18d20
Lung_Opacity-1420.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1420.png,Lung_Opacity/images/Lung_Opacity-1420.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1420.png,Lung_Opacity/masks/Lung_Opacity-1420.png,True,False,299,299,L,112.16,b54ae7e03dda747fdf2e885a241c685d
Lung_Opacity-1421.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1421.png,Lung_Opacity/images/Lung_Opacity-1421.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1421.png,Lung_Opacity/masks/Lung_Opacity-1421.png,True,False,299,299,L,49.16,513188b2e62a08968640547d0375ac45
Lung_Opacity-1422.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1422.png,Lung_Opacity/images/Lung_Opacity-1422.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1422.png,Lung_Opacity/masks/Lung_Opacity-1422.png,True,False,299,299,L,118.96,8b6b18226c604d358990926d9b5a4382
Lung_Opacity-1423.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1423.png,Lung_Opacity/images/Lung_Opacity-1423.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1423.png,Lung_Opacity/masks/Lung_Opacity-1423.png,True,False,299,299,L,100.23,ddce1628cfc68a3ad504867fcb7b512f
Lung_Opacity-1424.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1424.png,Lung_Opacity/images/Lung_Opacity-1424.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1424.png,Lung_Opacity/masks/Lung_Opacity-1424.png,True,False,299,299,L,125.36,604cf59251b8e24c4d59495bd651aeaf
Lung_Opacity-1425.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1425.png,Lung_Opacity/images/Lung_Opacity-1425.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1425.png,Lung_Opacity/masks/Lung_Opacity-1425.png,True,False,299,299,L,161.24,d22ee42023d2be5a913139c0eccec2a6
Lung_Opacity-1426.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1426.png,Lung_Opacity/images/Lung_Opacity-1426.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1426.png,Lung_Opacity/masks/Lung_Opacity-1426.png,True,False,299,299,L,122.08,b5066949d47f8641159200af6dffb762
Lung_Opacity-1427.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1427.png,Lung_Opacity/images/Lung_Opacity-1427.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1427.png,Lung_Opacity/masks/Lung_Opacity-1427.png,True,False,299,299,L,157.56,dc9a31aebbb21137694f7d647826187f
Lung_Opacity-1428.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1428.png,Lung_Opacity/images/Lung_Opacity-1428.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1428.png,Lung_Opacity/masks/Lung_Opacity-1428.png,True,False,299,299,L,151.1,d5ead87d15c40eef1c61c3eb4b74fc1d
Lung_Opacity-1429.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1429.png,Lung_Opacity/images/Lung_Opacity-1429.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1429.png,Lung_Opacity/masks/Lung_Opacity-1429.png,True,False,299,299,L,172.14,ce4e8ad08fd149b8fb7366cef1b0df76
Lung_Opacity-143.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-143.png,Lung_Opacity/images/Lung_Opacity-143.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-143.png,Lung_Opacity/masks/Lung_Opacity-143.png,True,False,299,299,L,131.93,14fccd8fe9642999580d50f105a602ff
Lung_Opacity-1430.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1430.png,Lung_Opacity/images/Lung_Opacity-1430.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1430.png,Lung_Opacity/masks/Lung_Opacity-1430.png,True,False,299,299,L,114.33,220b0d29ea48ffc6a7ce498799549d85
Lung_Opacity-1431.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1431.png,Lung_Opacity/images/Lung_Opacity-1431.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1431.png,Lung_Opacity/masks/Lung_Opacity-1431.png,True,False,299,299,L,121.8,6efafb8cd9a412cca9c2ac91022e8645
Lung_Opacity-1432.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1432.png,Lung_Opacity/images/Lung_Opacity-1432.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1432.png,Lung_Opacity/masks/Lung_Opacity-1432.png,True,False,299,299,L,84.42,50f96966249f16ebdb11546606210900
Lung_Opacity-1433.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1433.png,Lung_Opacity/images/Lung_Opacity-1433.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1433.png,Lung_Opacity/masks/Lung_Opacity-1433.png,True,False,299,299,L,147.41,08ce0336af5a0a01420defcf3213f1c9
Lung_Opacity-1434.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1434.png,Lung_Opacity/images/Lung_Opacity-1434.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1434.png,Lung_Opacity/masks/Lung_Opacity-1434.png,True,False,299,299,L,107.27,b3477ac39468ab70d605b2ec99c55fe8
Lung_Opacity-1435.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1435.png,Lung_Opacity/images/Lung_Opacity-1435.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1435.png,Lung_Opacity/masks/Lung_Opacity-1435.png,True,False,299,299,L,98.6,0c7225e296ab13e1ae5040e9c16d76ea
Lung_Opacity-1436.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1436.png,Lung_Opacity/images/Lung_Opacity-1436.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1436.png,Lung_Opacity/masks/Lung_Opacity-1436.png,True,False,299,299,L,109.18,2e151f6aa8b045ff867f1bd1671f6f68
Lung_Opacity-1437.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1437.png,Lung_Opacity/images/Lung_Opacity-1437.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1437.png,Lung_Opacity/masks/Lung_Opacity-1437.png,True,False,299,299,L,137.52,83f4d1da10d83b2521bf16cb4ae77c45
Lung_Opacity-1438.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1438.png,Lung_Opacity/images/Lung_Opacity-1438.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1438.png,Lung_Opacity/masks/Lung_Opacity-1438.png,True,False,299,299,L,112.38,11ecd48e7300e3527dc73fd482099372
Lung_Opacity-1439.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1439.png,Lung_Opacity/images/Lung_Opacity-1439.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1439.png,Lung_Opacity/masks/Lung_Opacity-1439.png,True,False,299,299,L,123.46,70ce3c7ee7a21e8cf45041f1743d9b09
Lung_Opacity-144.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-144.png,Lung_Opacity/images/Lung_Opacity-144.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-144.png,Lung_Opacity/masks/Lung_Opacity-144.png,True,False,299,299,L,147.59,2b834b57377fc2503815e7b48064c9aa
Lung_Opacity-1440.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1440.png,Lung_Opacity/images/Lung_Opacity-1440.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1440.png,Lung_Opacity/masks/Lung_Opacity-1440.png,True,False,299,299,L,136.8,5cc4b1d02df7792387bf34adcbc8ef49
Lung_Opacity-1441.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1441.png,Lung_Opacity/images/Lung_Opacity-1441.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1441.png,Lung_Opacity/masks/Lung_Opacity-1441.png,True,False,299,299,L,150.27,653faaf3db6348f1fd958f74c1797bbd
Lung_Opacity-1442.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1442.png,Lung_Opacity/images/Lung_Opacity-1442.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1442.png,Lung_Opacity/masks/Lung_Opacity-1442.png,True,False,299,299,L,128.68,731eedc3aca6a34e496f88c93643cde7
Lung_Opacity-1443.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1443.png,Lung_Opacity/images/Lung_Opacity-1443.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1443.png,Lung_Opacity/masks/Lung_Opacity-1443.png,True,False,299,299,L,146.64,94d1403f708e3e2db979c18a79a9766c
Lung_Opacity-1444.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1444.png,Lung_Opacity/images/Lung_Opacity-1444.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1444.png,Lung_Opacity/masks/Lung_Opacity-1444.png,True,False,299,299,L,159.26,40c5ecf6c5bb72aaa3ec227f09e99d15
Lung_Opacity-1445.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1445.png,Lung_Opacity/images/Lung_Opacity-1445.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1445.png,Lung_Opacity/masks/Lung_Opacity-1445.png,True,False,299,299,L,115.9,26e53772e423f60e969f95783d966317
Lung_Opacity-1446.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1446.png,Lung_Opacity/images/Lung_Opacity-1446.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1446.png,Lung_Opacity/masks/Lung_Opacity-1446.png,True,False,299,299,L,170.06,5918857f00bbb9698922d173a1cbea33
Lung_Opacity-1447.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1447.png,Lung_Opacity/images/Lung_Opacity-1447.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1447.png,Lung_Opacity/masks/Lung_Opacity-1447.png,True,False,299,299,L,121.6,5c1d8ede812bc79a8298a9f82f48ba19
Lung_Opacity-1448.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1448.png,Lung_Opacity/images/Lung_Opacity-1448.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1448.png,Lung_Opacity/masks/Lung_Opacity-1448.png,True,False,299,299,L,122.61,f7f0555dbc116598c60ef3406acea844
Lung_Opacity-1449.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1449.png,Lung_Opacity/images/Lung_Opacity-1449.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1449.png,Lung_Opacity/masks/Lung_Opacity-1449.png,True,False,299,299,L,131.18,d2666d57470530b754c102106f0c8250
Lung_Opacity-145.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-145.png,Lung_Opacity/images/Lung_Opacity-145.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-145.png,Lung_Opacity/masks/Lung_Opacity-145.png,True,False,299,299,L,126.69,20d523b3aef72ae5be2263bf0c398bee
Lung_Opacity-1450.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1450.png,Lung_Opacity/images/Lung_Opacity-1450.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1450.png,Lung_Opacity/masks/Lung_Opacity-1450.png,True,False,299,299,L,105.54,a387692569c62e60e677af28025fff13
Lung_Opacity-1451.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1451.png,Lung_Opacity/images/Lung_Opacity-1451.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1451.png,Lung_Opacity/masks/Lung_Opacity-1451.png,True,False,299,299,L,168.76,6f53596d1c61edb57a1ae100bbf088ad
Lung_Opacity-1452.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1452.png,Lung_Opacity/images/Lung_Opacity-1452.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1452.png,Lung_Opacity/masks/Lung_Opacity-1452.png,True,False,299,299,L,134.49,65b069808a49b888984d8a3b7bc5e802
Lung_Opacity-1453.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1453.png,Lung_Opacity/images/Lung_Opacity-1453.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1453.png,Lung_Opacity/masks/Lung_Opacity-1453.png,True,False,299,299,L,164.3,8ec0a3ab8c9885dac33c933bdb9264a1
Lung_Opacity-1454.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1454.png,Lung_Opacity/images/Lung_Opacity-1454.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1454.png,Lung_Opacity/masks/Lung_Opacity-1454.png,True,False,299,299,L,105.5,ad44de0ab0c46e08d10295c22423b6ee
Lung_Opacity-1455.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1455.png,Lung_Opacity/images/Lung_Opacity-1455.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1455.png,Lung_Opacity/masks/Lung_Opacity-1455.png,True,False,299,299,L,125.26,40b0b27b7e52500b1e98224aa8150492
Lung_Opacity-1456.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1456.png,Lung_Opacity/images/Lung_Opacity-1456.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1456.png,Lung_Opacity/masks/Lung_Opacity-1456.png,True,False,299,299,L,111.34,b5b965a8e4dd3da4ee246436e7f619f5
Lung_Opacity-1457.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1457.png,Lung_Opacity/images/Lung_Opacity-1457.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1457.png,Lung_Opacity/masks/Lung_Opacity-1457.png,True,False,299,299,L,150.78,ff8bd30bb44d81bb33b5b66b8385b357
Lung_Opacity-1458.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1458.png,Lung_Opacity/images/Lung_Opacity-1458.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1458.png,Lung_Opacity/masks/Lung_Opacity-1458.png,True,False,299,299,L,153.08,e7491903f6afca7e1ae6280c9a64e650
Lung_Opacity-1459.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1459.png,Lung_Opacity/images/Lung_Opacity-1459.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1459.png,Lung_Opacity/masks/Lung_Opacity-1459.png,True,False,299,299,L,165.66,d9ff652823162fbba0e840e8e0fecb98
Lung_Opacity-146.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-146.png,Lung_Opacity/images/Lung_Opacity-146.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-146.png,Lung_Opacity/masks/Lung_Opacity-146.png,True,False,299,299,L,106.54,d41480e0047d7ccc53a2a9d222574c78
Lung_Opacity-1460.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1460.png,Lung_Opacity/images/Lung_Opacity-1460.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1460.png,Lung_Opacity/masks/Lung_Opacity-1460.png,True,False,299,299,L,110.24,00b30166aa75b0e163bf85be28fe74ff
Lung_Opacity-1461.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1461.png,Lung_Opacity/images/Lung_Opacity-1461.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1461.png,Lung_Opacity/masks/Lung_Opacity-1461.png,True,False,299,299,L,120.69,bbbf80c087e845bbcf4e5108640c2d8f
Lung_Opacity-1462.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1462.png,Lung_Opacity/images/Lung_Opacity-1462.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1462.png,Lung_Opacity/masks/Lung_Opacity-1462.png,True,False,299,299,L,94.86,13b1f2edc05796b02893c31baf432c1e
Lung_Opacity-1463.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1463.png,Lung_Opacity/images/Lung_Opacity-1463.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1463.png,Lung_Opacity/masks/Lung_Opacity-1463.png,True,False,299,299,L,140.39,3793201d75ee47bd1a04bb1efcc4a62e
Lung_Opacity-1464.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1464.png,Lung_Opacity/images/Lung_Opacity-1464.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1464.png,Lung_Opacity/masks/Lung_Opacity-1464.png,True,False,299,299,L,121.9,1af6e482c69c703d8a7938704a8974dd
Lung_Opacity-1465.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1465.png,Lung_Opacity/images/Lung_Opacity-1465.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1465.png,Lung_Opacity/masks/Lung_Opacity-1465.png,True,False,299,299,L,144.43,22a817a67bd80482bc858f2bbf5b8ce9
Lung_Opacity-1466.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1466.png,Lung_Opacity/images/Lung_Opacity-1466.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1466.png,Lung_Opacity/masks/Lung_Opacity-1466.png,True,False,299,299,L,149.05,5bb46a2ecda62e7ae357531a6db34b71
Lung_Opacity-1467.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1467.png,Lung_Opacity/images/Lung_Opacity-1467.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1467.png,Lung_Opacity/masks/Lung_Opacity-1467.png,True,False,299,299,L,114.22,bb0bb2829cb1b94ab5e8a419a97a8e08
Lung_Opacity-1468.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1468.png,Lung_Opacity/images/Lung_Opacity-1468.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1468.png,Lung_Opacity/masks/Lung_Opacity-1468.png,True,False,299,299,L,101.12,fa187362a5fb159ff61571b16b39ed94
Lung_Opacity-1469.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1469.png,Lung_Opacity/images/Lung_Opacity-1469.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1469.png,Lung_Opacity/masks/Lung_Opacity-1469.png,True,False,299,299,L,117.04,3c7b438627b333c61c8856624e9bebb7
Lung_Opacity-147.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-147.png,Lung_Opacity/images/Lung_Opacity-147.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-147.png,Lung_Opacity/masks/Lung_Opacity-147.png,True,False,299,299,L,168.03,7713931111841a00b530e61511a0d8e5
Lung_Opacity-1470.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1470.png,Lung_Opacity/images/Lung_Opacity-1470.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1470.png,Lung_Opacity/masks/Lung_Opacity-1470.png,True,False,299,299,L,131.73,a037f72f8274070942d5fa2b29cd6001
Lung_Opacity-1471.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1471.png,Lung_Opacity/images/Lung_Opacity-1471.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1471.png,Lung_Opacity/masks/Lung_Opacity-1471.png,True,False,299,299,L,125.11,37ad00aa734ff15d2b2cf10de1574afd
Lung_Opacity-1472.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1472.png,Lung_Opacity/images/Lung_Opacity-1472.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1472.png,Lung_Opacity/masks/Lung_Opacity-1472.png,True,False,299,299,L,95.06,4c874f027710e34c37221e0ce36a06bd
Lung_Opacity-1473.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1473.png,Lung_Opacity/images/Lung_Opacity-1473.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1473.png,Lung_Opacity/masks/Lung_Opacity-1473.png,True,False,299,299,L,113.59,243c41ae86c19a5695f8678cd341149c
Lung_Opacity-1474.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1474.png,Lung_Opacity/images/Lung_Opacity-1474.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1474.png,Lung_Opacity/masks/Lung_Opacity-1474.png,True,False,299,299,L,124.5,a07bc1059c276ce98e30669306feac8e
Lung_Opacity-1475.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1475.png,Lung_Opacity/images/Lung_Opacity-1475.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1475.png,Lung_Opacity/masks/Lung_Opacity-1475.png,True,False,299,299,L,120.19,7d532bf9a4a397a619c035610c94f126
Lung_Opacity-1476.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1476.png,Lung_Opacity/images/Lung_Opacity-1476.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1476.png,Lung_Opacity/masks/Lung_Opacity-1476.png,True,False,299,299,L,127.95,5cf7200a455e1bc0a408fbdec6d880bb
Lung_Opacity-1477.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1477.png,Lung_Opacity/images/Lung_Opacity-1477.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1477.png,Lung_Opacity/masks/Lung_Opacity-1477.png,True,False,299,299,L,119.57,dbdf2e55c06085794cdc5acf6e33d369
Lung_Opacity-1478.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1478.png,Lung_Opacity/images/Lung_Opacity-1478.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1478.png,Lung_Opacity/masks/Lung_Opacity-1478.png,True,False,299,299,L,99.45,2bd1f171894224880a8290f684c90e20
Lung_Opacity-1479.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1479.png,Lung_Opacity/images/Lung_Opacity-1479.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1479.png,Lung_Opacity/masks/Lung_Opacity-1479.png,True,False,299,299,L,103.77,62382088f51fb8bde775820765a731cf
Lung_Opacity-148.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-148.png,Lung_Opacity/images/Lung_Opacity-148.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-148.png,Lung_Opacity/masks/Lung_Opacity-148.png,True,False,299,299,L,106.67,6b8e234b0b478d2acd887289b8d36c1a
Lung_Opacity-1480.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1480.png,Lung_Opacity/images/Lung_Opacity-1480.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1480.png,Lung_Opacity/masks/Lung_Opacity-1480.png,True,False,299,299,L,139.67,39e710d0a659a2bb580e6be33e16e0ed
Lung_Opacity-1481.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1481.png,Lung_Opacity/images/Lung_Opacity-1481.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1481.png,Lung_Opacity/masks/Lung_Opacity-1481.png,True,False,299,299,L,107.7,933c6abc6743897c89b0d977deaa2a05
Lung_Opacity-1482.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1482.png,Lung_Opacity/images/Lung_Opacity-1482.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1482.png,Lung_Opacity/masks/Lung_Opacity-1482.png,True,False,299,299,L,148.1,0b041b7810c85b58df68ffe9bc7c73b3
Lung_Opacity-1483.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1483.png,Lung_Opacity/images/Lung_Opacity-1483.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1483.png,Lung_Opacity/masks/Lung_Opacity-1483.png,True,False,299,299,L,123.04,adee6f53dc51bf5e4741dd3a20615c66
Lung_Opacity-1484.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1484.png,Lung_Opacity/images/Lung_Opacity-1484.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1484.png,Lung_Opacity/masks/Lung_Opacity-1484.png,True,False,299,299,L,144.42,d8b4b31e4cc4f980accbea46217a89a5
Lung_Opacity-1485.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1485.png,Lung_Opacity/images/Lung_Opacity-1485.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1485.png,Lung_Opacity/masks/Lung_Opacity-1485.png,True,False,299,299,L,125.42,39fca408ce6503eee7ae6f2556e86a39
Lung_Opacity-1486.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1486.png,Lung_Opacity/images/Lung_Opacity-1486.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1486.png,Lung_Opacity/masks/Lung_Opacity-1486.png,True,False,299,299,L,98.54,3075a5feff5189282c28a88a64953ae0
Lung_Opacity-1487.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1487.png,Lung_Opacity/images/Lung_Opacity-1487.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1487.png,Lung_Opacity/masks/Lung_Opacity-1487.png,True,False,299,299,L,84.65,c166fb312668a518cedc1c097c63ce8a
Lung_Opacity-1488.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1488.png,Lung_Opacity/images/Lung_Opacity-1488.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1488.png,Lung_Opacity/masks/Lung_Opacity-1488.png,True,False,299,299,L,135.37,983b8f3bb3a762302b235d336250db96
Lung_Opacity-1489.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1489.png,Lung_Opacity/images/Lung_Opacity-1489.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1489.png,Lung_Opacity/masks/Lung_Opacity-1489.png,True,False,299,299,L,107.4,6a53fec103a52478985931b04031a6e6
Lung_Opacity-149.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-149.png,Lung_Opacity/images/Lung_Opacity-149.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-149.png,Lung_Opacity/masks/Lung_Opacity-149.png,True,False,299,299,L,63.97,571b4d94a27980e5211bc32679e73961
Lung_Opacity-1490.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1490.png,Lung_Opacity/images/Lung_Opacity-1490.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1490.png,Lung_Opacity/masks/Lung_Opacity-1490.png,True,False,299,299,L,110.1,9cbc62762c2ace7d3ef6639ce55c92bf
Lung_Opacity-1491.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1491.png,Lung_Opacity/images/Lung_Opacity-1491.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1491.png,Lung_Opacity/masks/Lung_Opacity-1491.png,True,False,299,299,L,93.48,cd60e43263019550c79f7b0d43cd971a
Lung_Opacity-1492.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1492.png,Lung_Opacity/images/Lung_Opacity-1492.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1492.png,Lung_Opacity/masks/Lung_Opacity-1492.png,True,False,299,299,L,119.87,c7d9c372d64c91d041cfe6afcb55260d
Lung_Opacity-1493.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1493.png,Lung_Opacity/images/Lung_Opacity-1493.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1493.png,Lung_Opacity/masks/Lung_Opacity-1493.png,True,False,299,299,L,97.11,2345f84566142081e253a9b7a112afc1
Lung_Opacity-1494.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1494.png,Lung_Opacity/images/Lung_Opacity-1494.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1494.png,Lung_Opacity/masks/Lung_Opacity-1494.png,True,False,299,299,L,125.88,071c0c1f8d43333b56d635cd1b488303
Lung_Opacity-1495.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1495.png,Lung_Opacity/images/Lung_Opacity-1495.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1495.png,Lung_Opacity/masks/Lung_Opacity-1495.png,True,False,299,299,L,140.02,f76a2fdb766ec70ee1acf0ce891fd39a
Lung_Opacity-1496.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1496.png,Lung_Opacity/images/Lung_Opacity-1496.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1496.png,Lung_Opacity/masks/Lung_Opacity-1496.png,True,False,299,299,L,148.03,c63fb169c8c25f55c177aae03c7c1aca
Lung_Opacity-1497.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1497.png,Lung_Opacity/images/Lung_Opacity-1497.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1497.png,Lung_Opacity/masks/Lung_Opacity-1497.png,True,False,299,299,L,109.13,8ad88d10081d2cd4cc654fbbe9d7d851
Lung_Opacity-1498.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1498.png,Lung_Opacity/images/Lung_Opacity-1498.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1498.png,Lung_Opacity/masks/Lung_Opacity-1498.png,True,False,299,299,L,94.16,25acd2f018330f72f0e7ef84f18b8c74
Lung_Opacity-1499.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1499.png,Lung_Opacity/images/Lung_Opacity-1499.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1499.png,Lung_Opacity/masks/Lung_Opacity-1499.png,True,False,299,299,L,108.15,17576637b96d68a065ac2b1e172b1c0c
Lung_Opacity-15.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-15.png,Lung_Opacity/images/Lung_Opacity-15.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-15.png,Lung_Opacity/masks/Lung_Opacity-15.png,True,False,299,299,L,98.11,b81f3b5ebba70e901514e41b95c4769d
Lung_Opacity-150.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-150.png,Lung_Opacity/images/Lung_Opacity-150.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-150.png,Lung_Opacity/masks/Lung_Opacity-150.png,True,False,299,299,L,122.86,b728c2eae2077e45ac3ef99bbd95f73e
Lung_Opacity-1500.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1500.png,Lung_Opacity/images/Lung_Opacity-1500.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1500.png,Lung_Opacity/masks/Lung_Opacity-1500.png,True,False,299,299,L,130.38,7a61953180cea5aae01ce9aa8cd2e7f0
Lung_Opacity-1501.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1501.png,Lung_Opacity/images/Lung_Opacity-1501.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1501.png,Lung_Opacity/masks/Lung_Opacity-1501.png,True,False,299,299,L,126.62,b0da7c0538f87932c08c951fa304ed6c
Lung_Opacity-1502.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1502.png,Lung_Opacity/images/Lung_Opacity-1502.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1502.png,Lung_Opacity/masks/Lung_Opacity-1502.png,True,False,299,299,L,160.0,4369426251c360c7a75e0afc52a3f5b7
Lung_Opacity-1503.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1503.png,Lung_Opacity/images/Lung_Opacity-1503.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1503.png,Lung_Opacity/masks/Lung_Opacity-1503.png,True,False,299,299,L,114.13,fa54aa450e3cf1ac400202080e82851e
Lung_Opacity-1504.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1504.png,Lung_Opacity/images/Lung_Opacity-1504.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1504.png,Lung_Opacity/masks/Lung_Opacity-1504.png,True,False,299,299,L,141.61,f67db47668199e6f5b9f297974921140
Lung_Opacity-1505.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1505.png,Lung_Opacity/images/Lung_Opacity-1505.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1505.png,Lung_Opacity/masks/Lung_Opacity-1505.png,True,False,299,299,L,133.75,99a9b0519a8e68edae8a38682749c0e6
Lung_Opacity-1506.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1506.png,Lung_Opacity/images/Lung_Opacity-1506.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1506.png,Lung_Opacity/masks/Lung_Opacity-1506.png,True,False,299,299,L,127.41,d3a14ca8dc790b0db45a5e0377da5b2d
Lung_Opacity-1507.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1507.png,Lung_Opacity/images/Lung_Opacity-1507.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1507.png,Lung_Opacity/masks/Lung_Opacity-1507.png,True,False,299,299,L,142.95,5c330367bdac3e5eb45e545277f7d109
Lung_Opacity-1508.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1508.png,Lung_Opacity/images/Lung_Opacity-1508.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1508.png,Lung_Opacity/masks/Lung_Opacity-1508.png,True,False,299,299,L,98.36,4c55fc470390a630d305a22d109c21e1
Lung_Opacity-1509.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1509.png,Lung_Opacity/images/Lung_Opacity-1509.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1509.png,Lung_Opacity/masks/Lung_Opacity-1509.png,True,False,299,299,L,151.58,6b5a7011cd88513c27c21dd59bff45e2
Lung_Opacity-151.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-151.png,Lung_Opacity/images/Lung_Opacity-151.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-151.png,Lung_Opacity/masks/Lung_Opacity-151.png,True,False,299,299,L,116.54,edb0bd282c6b1ab436368c56dc8575c0
Lung_Opacity-1510.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1510.png,Lung_Opacity/images/Lung_Opacity-1510.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1510.png,Lung_Opacity/masks/Lung_Opacity-1510.png,True,False,299,299,L,179.87,b8ba2da3c921955c3ca485351feb514a
Lung_Opacity-1511.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1511.png,Lung_Opacity/images/Lung_Opacity-1511.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1511.png,Lung_Opacity/masks/Lung_Opacity-1511.png,True,False,299,299,L,127.16,56ca5183a9eb6abb879dba8c56e8bb84
Lung_Opacity-1512.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1512.png,Lung_Opacity/images/Lung_Opacity-1512.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1512.png,Lung_Opacity/masks/Lung_Opacity-1512.png,True,False,299,299,L,128.96,3f48c11423aa95b35489f9a5090187c8
Lung_Opacity-1513.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1513.png,Lung_Opacity/images/Lung_Opacity-1513.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1513.png,Lung_Opacity/masks/Lung_Opacity-1513.png,True,False,299,299,L,109.64,66230a690ce87fbfc095c8d0421937df
Lung_Opacity-1514.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1514.png,Lung_Opacity/images/Lung_Opacity-1514.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1514.png,Lung_Opacity/masks/Lung_Opacity-1514.png,True,False,299,299,L,175.57,ddf28b0efb6fbd298bfd3bb3e754b2eb
Lung_Opacity-1515.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1515.png,Lung_Opacity/images/Lung_Opacity-1515.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1515.png,Lung_Opacity/masks/Lung_Opacity-1515.png,True,False,299,299,L,116.87,e1dc5da4809b5469097e20f57fb53ed0
Lung_Opacity-1516.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1516.png,Lung_Opacity/images/Lung_Opacity-1516.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1516.png,Lung_Opacity/masks/Lung_Opacity-1516.png,True,False,299,299,L,106.74,bd71a847f232ec1f7a3cef84367cb90e
Lung_Opacity-1517.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1517.png,Lung_Opacity/images/Lung_Opacity-1517.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1517.png,Lung_Opacity/masks/Lung_Opacity-1517.png,True,False,299,299,L,144.09,5ae5a6779c49f51ff9ef11712c21167a
Lung_Opacity-1518.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1518.png,Lung_Opacity/images/Lung_Opacity-1518.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1518.png,Lung_Opacity/masks/Lung_Opacity-1518.png,True,False,299,299,L,96.59,2a2b5e0e4eeb4e32ba30e2b9ac38c0c4
Lung_Opacity-1519.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1519.png,Lung_Opacity/images/Lung_Opacity-1519.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1519.png,Lung_Opacity/masks/Lung_Opacity-1519.png,True,False,299,299,L,106.28,27723eb552d0343ecdd6c21627d0f03c
Lung_Opacity-152.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-152.png,Lung_Opacity/images/Lung_Opacity-152.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-152.png,Lung_Opacity/masks/Lung_Opacity-152.png,True,False,299,299,L,111.19,9e6d927ae8bf85229ca1e0a53a6e3b67
Lung_Opacity-1520.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1520.png,Lung_Opacity/images/Lung_Opacity-1520.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1520.png,Lung_Opacity/masks/Lung_Opacity-1520.png,True,False,299,299,L,150.64,3789df340a714797ce3b5cc01b96030c
Lung_Opacity-1521.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1521.png,Lung_Opacity/images/Lung_Opacity-1521.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1521.png,Lung_Opacity/masks/Lung_Opacity-1521.png,True,False,299,299,L,106.18,374dcd12a55eb966d57c3a2159370971
Lung_Opacity-1522.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1522.png,Lung_Opacity/images/Lung_Opacity-1522.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1522.png,Lung_Opacity/masks/Lung_Opacity-1522.png,True,False,299,299,L,100.43,ad18967d2702fd7a711a69fe918b58cc
Lung_Opacity-1523.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1523.png,Lung_Opacity/images/Lung_Opacity-1523.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1523.png,Lung_Opacity/masks/Lung_Opacity-1523.png,True,False,299,299,L,114.48,239aa5e8b9ecc7dc37274b2861462fe7
Lung_Opacity-1524.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1524.png,Lung_Opacity/images/Lung_Opacity-1524.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1524.png,Lung_Opacity/masks/Lung_Opacity-1524.png,True,False,299,299,L,130.77,9cdafe024507502c8ed9f4a0cbe175a6
Lung_Opacity-1525.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1525.png,Lung_Opacity/images/Lung_Opacity-1525.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1525.png,Lung_Opacity/masks/Lung_Opacity-1525.png,True,False,299,299,L,123.48,2405ad9ee78f8dbb5e9c7a08d45f3590
Lung_Opacity-1526.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1526.png,Lung_Opacity/images/Lung_Opacity-1526.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1526.png,Lung_Opacity/masks/Lung_Opacity-1526.png,True,False,299,299,L,83.87,7effbb7ef4e2c7ad85280eec500f752b
Lung_Opacity-1527.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1527.png,Lung_Opacity/images/Lung_Opacity-1527.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1527.png,Lung_Opacity/masks/Lung_Opacity-1527.png,True,False,299,299,L,171.02,6be72154547baa2b4440d1f4ba55676b
Lung_Opacity-1528.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1528.png,Lung_Opacity/images/Lung_Opacity-1528.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1528.png,Lung_Opacity/masks/Lung_Opacity-1528.png,True,False,299,299,L,169.25,a9a86279f2fd5752d6064ced368126a2
Lung_Opacity-1529.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1529.png,Lung_Opacity/images/Lung_Opacity-1529.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1529.png,Lung_Opacity/masks/Lung_Opacity-1529.png,True,False,299,299,L,172.84,66c3090f4751c29cd8f1280f8278d05d
Lung_Opacity-153.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-153.png,Lung_Opacity/images/Lung_Opacity-153.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-153.png,Lung_Opacity/masks/Lung_Opacity-153.png,True,False,299,299,L,112.26,b6a03ef9bb4b33bab5b32056875d6a04
Lung_Opacity-1530.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1530.png,Lung_Opacity/images/Lung_Opacity-1530.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1530.png,Lung_Opacity/masks/Lung_Opacity-1530.png,True,False,299,299,L,148.14,d4b3a38fbec011111bb8558a666318d7
Lung_Opacity-1531.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1531.png,Lung_Opacity/images/Lung_Opacity-1531.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1531.png,Lung_Opacity/masks/Lung_Opacity-1531.png,True,False,299,299,L,137.75,1ee81880bf516c162a9e9a099fb7385c
Lung_Opacity-1532.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1532.png,Lung_Opacity/images/Lung_Opacity-1532.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1532.png,Lung_Opacity/masks/Lung_Opacity-1532.png,True,False,299,299,L,168.83,4dfd4057665b44c25c69fef55fc9d0bc
Lung_Opacity-1533.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1533.png,Lung_Opacity/images/Lung_Opacity-1533.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1533.png,Lung_Opacity/masks/Lung_Opacity-1533.png,True,False,299,299,L,122.3,023c145bd0f52536b178dba311a84dbc
Lung_Opacity-1534.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1534.png,Lung_Opacity/images/Lung_Opacity-1534.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1534.png,Lung_Opacity/masks/Lung_Opacity-1534.png,True,False,299,299,L,115.59,1774f992adf05ad27e0ba91c0b9e70f2
Lung_Opacity-1535.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1535.png,Lung_Opacity/images/Lung_Opacity-1535.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1535.png,Lung_Opacity/masks/Lung_Opacity-1535.png,True,False,299,299,L,106.74,7312f28e8c4e79aeee1339083f1a6ee4
Lung_Opacity-1536.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1536.png,Lung_Opacity/images/Lung_Opacity-1536.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1536.png,Lung_Opacity/masks/Lung_Opacity-1536.png,True,False,299,299,L,130.29,5cd3f9bb0555f65e434115bf93e2ed3a
Lung_Opacity-1537.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1537.png,Lung_Opacity/images/Lung_Opacity-1537.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1537.png,Lung_Opacity/masks/Lung_Opacity-1537.png,True,False,299,299,L,112.48,147c0e7de82ac06d32f25de05458487a
Lung_Opacity-1538.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1538.png,Lung_Opacity/images/Lung_Opacity-1538.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1538.png,Lung_Opacity/masks/Lung_Opacity-1538.png,True,False,299,299,L,121.44,a699cd01066b4bcdcc477927bd869b43
Lung_Opacity-1539.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1539.png,Lung_Opacity/images/Lung_Opacity-1539.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1539.png,Lung_Opacity/masks/Lung_Opacity-1539.png,True,False,299,299,L,93.96,7ce5e1172d61f02fd48d91fc7c675b4f
Lung_Opacity-154.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-154.png,Lung_Opacity/images/Lung_Opacity-154.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-154.png,Lung_Opacity/masks/Lung_Opacity-154.png,True,False,299,299,L,119.82,76ffb00dbee721a98d751cc639afadf6
Lung_Opacity-1540.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1540.png,Lung_Opacity/images/Lung_Opacity-1540.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1540.png,Lung_Opacity/masks/Lung_Opacity-1540.png,True,False,299,299,L,171.28,505c039c7eb17cfce93aaca8e66c64a0
Lung_Opacity-1541.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1541.png,Lung_Opacity/images/Lung_Opacity-1541.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1541.png,Lung_Opacity/masks/Lung_Opacity-1541.png,True,False,299,299,L,144.34,6c9639864b3d04cf541b433a6bdef82c
Lung_Opacity-1542.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1542.png,Lung_Opacity/images/Lung_Opacity-1542.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1542.png,Lung_Opacity/masks/Lung_Opacity-1542.png,True,False,299,299,L,139.82,0b11ef8df050d21b6b3cd58f7150dbe3
Lung_Opacity-1543.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1543.png,Lung_Opacity/images/Lung_Opacity-1543.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1543.png,Lung_Opacity/masks/Lung_Opacity-1543.png,True,False,299,299,L,165.42,b4132468de9d5692e3b939cac37f1662
Lung_Opacity-1544.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1544.png,Lung_Opacity/images/Lung_Opacity-1544.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1544.png,Lung_Opacity/masks/Lung_Opacity-1544.png,True,False,299,299,L,121.39,66ce87b2043b3821a242c63fb59feed5
Lung_Opacity-1545.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1545.png,Lung_Opacity/images/Lung_Opacity-1545.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1545.png,Lung_Opacity/masks/Lung_Opacity-1545.png,True,False,299,299,L,108.06,43d433fca1c1838ae684f2d87be57b34
Lung_Opacity-1546.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1546.png,Lung_Opacity/images/Lung_Opacity-1546.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1546.png,Lung_Opacity/masks/Lung_Opacity-1546.png,True,False,299,299,L,138.35,17d18ee2799d01c546cc2687a08d61bd
Lung_Opacity-1547.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1547.png,Lung_Opacity/images/Lung_Opacity-1547.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1547.png,Lung_Opacity/masks/Lung_Opacity-1547.png,True,False,299,299,L,137.64,b0c2c816557a0023400b2b04c06960e7
Lung_Opacity-1548.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1548.png,Lung_Opacity/images/Lung_Opacity-1548.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1548.png,Lung_Opacity/masks/Lung_Opacity-1548.png,True,False,299,299,L,129.7,4af134e610faca7618f712fcc384a2d5
Lung_Opacity-1549.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1549.png,Lung_Opacity/images/Lung_Opacity-1549.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1549.png,Lung_Opacity/masks/Lung_Opacity-1549.png,True,False,299,299,L,162.01,eedc9fdee83320845931c53c0aa98f1d
Lung_Opacity-155.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-155.png,Lung_Opacity/images/Lung_Opacity-155.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-155.png,Lung_Opacity/masks/Lung_Opacity-155.png,True,False,299,299,L,166.62,6322310d6740c3ed8a3839a216433fe7
Lung_Opacity-1550.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1550.png,Lung_Opacity/images/Lung_Opacity-1550.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1550.png,Lung_Opacity/masks/Lung_Opacity-1550.png,True,False,299,299,L,145.1,107a878cd86d92e6754c6f549ae6f173
Lung_Opacity-1551.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1551.png,Lung_Opacity/images/Lung_Opacity-1551.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1551.png,Lung_Opacity/masks/Lung_Opacity-1551.png,True,False,299,299,L,179.77,bc67d66e83c26f9f8af3216688ff5cf1
Lung_Opacity-1552.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1552.png,Lung_Opacity/images/Lung_Opacity-1552.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1552.png,Lung_Opacity/masks/Lung_Opacity-1552.png,True,False,299,299,L,162.05,41db12627676acdd911b94f3cca7ecc5
Lung_Opacity-1553.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1553.png,Lung_Opacity/images/Lung_Opacity-1553.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1553.png,Lung_Opacity/masks/Lung_Opacity-1553.png,True,False,299,299,L,120.23,368e948caa5ebf721c17d86dc5e92052
Lung_Opacity-1554.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1554.png,Lung_Opacity/images/Lung_Opacity-1554.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1554.png,Lung_Opacity/masks/Lung_Opacity-1554.png,True,False,299,299,L,159.04,1c548330681a7ad799aea8050e3cba58
Lung_Opacity-1555.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1555.png,Lung_Opacity/images/Lung_Opacity-1555.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1555.png,Lung_Opacity/masks/Lung_Opacity-1555.png,True,False,299,299,L,106.52,9ffa622c72a79342e70647e3f54a5c4d
Lung_Opacity-1556.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1556.png,Lung_Opacity/images/Lung_Opacity-1556.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1556.png,Lung_Opacity/masks/Lung_Opacity-1556.png,True,False,299,299,L,140.29,76762b2105a0babdf88c0626c17b92cd
Lung_Opacity-1557.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1557.png,Lung_Opacity/images/Lung_Opacity-1557.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1557.png,Lung_Opacity/masks/Lung_Opacity-1557.png,True,False,299,299,L,96.24,609f653609d9deebd5941f8bc046d471
Lung_Opacity-1558.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1558.png,Lung_Opacity/images/Lung_Opacity-1558.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1558.png,Lung_Opacity/masks/Lung_Opacity-1558.png,True,False,299,299,L,176.97,ef347a645951912f3baeb783ca720a2f
Lung_Opacity-1559.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1559.png,Lung_Opacity/images/Lung_Opacity-1559.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1559.png,Lung_Opacity/masks/Lung_Opacity-1559.png,True,False,299,299,L,138.27,cd9ccaf27165813da4652e5ba3ee78bb
Lung_Opacity-156.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-156.png,Lung_Opacity/images/Lung_Opacity-156.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-156.png,Lung_Opacity/masks/Lung_Opacity-156.png,True,False,299,299,L,106.83,43ea040ccfa34cced4050ce502dfc61d
Lung_Opacity-1560.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1560.png,Lung_Opacity/images/Lung_Opacity-1560.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1560.png,Lung_Opacity/masks/Lung_Opacity-1560.png,True,False,299,299,L,96.53,b3d8bf28a3c31645c75abacc8c18c26c
Lung_Opacity-1561.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1561.png,Lung_Opacity/images/Lung_Opacity-1561.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1561.png,Lung_Opacity/masks/Lung_Opacity-1561.png,True,False,299,299,L,129.47,11d1ed413c9353b4ca4b1ccfb0efb147
Lung_Opacity-1562.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1562.png,Lung_Opacity/images/Lung_Opacity-1562.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1562.png,Lung_Opacity/masks/Lung_Opacity-1562.png,True,False,299,299,L,132.43,04637c420b449101935e0b2534d127d9
Lung_Opacity-1563.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1563.png,Lung_Opacity/images/Lung_Opacity-1563.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1563.png,Lung_Opacity/masks/Lung_Opacity-1563.png,True,False,299,299,L,113.45,ab1c2ada2fa4715be38019d26413c589
Lung_Opacity-1564.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1564.png,Lung_Opacity/images/Lung_Opacity-1564.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1564.png,Lung_Opacity/masks/Lung_Opacity-1564.png,True,False,299,299,L,177.96,86acaa4c8aadfaf6af67ec73497399ad
Lung_Opacity-1565.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1565.png,Lung_Opacity/images/Lung_Opacity-1565.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1565.png,Lung_Opacity/masks/Lung_Opacity-1565.png,True,False,299,299,L,111.08,9e305d2970ba3aabb778f3793104f1d3
Lung_Opacity-1566.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1566.png,Lung_Opacity/images/Lung_Opacity-1566.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1566.png,Lung_Opacity/masks/Lung_Opacity-1566.png,True,False,299,299,L,143.46,3a7363ca424005f4283e3f52c2d57377
Lung_Opacity-1567.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1567.png,Lung_Opacity/images/Lung_Opacity-1567.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1567.png,Lung_Opacity/masks/Lung_Opacity-1567.png,True,False,299,299,L,118.98,717f526210ffd7f56633e331e4c32a1a
Lung_Opacity-1568.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1568.png,Lung_Opacity/images/Lung_Opacity-1568.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1568.png,Lung_Opacity/masks/Lung_Opacity-1568.png,True,False,299,299,L,91.66,64486064ff95f92548d8bf90cb284b31
Lung_Opacity-1569.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1569.png,Lung_Opacity/images/Lung_Opacity-1569.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1569.png,Lung_Opacity/masks/Lung_Opacity-1569.png,True,False,299,299,L,180.27,955c894129168c791b28d49cb8ea98da
Lung_Opacity-157.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-157.png,Lung_Opacity/images/Lung_Opacity-157.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-157.png,Lung_Opacity/masks/Lung_Opacity-157.png,True,False,299,299,L,134.02,efa232fc99964ad0d21030ec0fca0e36
Lung_Opacity-1570.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1570.png,Lung_Opacity/images/Lung_Opacity-1570.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1570.png,Lung_Opacity/masks/Lung_Opacity-1570.png,True,False,299,299,L,111.1,bdc4db61194c9b7704ce4b09e073b4f5
Lung_Opacity-1571.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1571.png,Lung_Opacity/images/Lung_Opacity-1571.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1571.png,Lung_Opacity/masks/Lung_Opacity-1571.png,True,False,299,299,L,121.64,d33aa77ada4ac55f025ad596bd19bca4
Lung_Opacity-1572.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1572.png,Lung_Opacity/images/Lung_Opacity-1572.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1572.png,Lung_Opacity/masks/Lung_Opacity-1572.png,True,False,299,299,L,112.05,ce716406bfcc5f95ad73252c428451c3
Lung_Opacity-1573.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1573.png,Lung_Opacity/images/Lung_Opacity-1573.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1573.png,Lung_Opacity/masks/Lung_Opacity-1573.png,True,False,299,299,L,117.07,928abf4e7227e79fb14bcd5781f4781b
Lung_Opacity-1574.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1574.png,Lung_Opacity/images/Lung_Opacity-1574.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1574.png,Lung_Opacity/masks/Lung_Opacity-1574.png,True,False,299,299,L,113.6,30e0f4241ff5146edcbbcfe4c272087d
Lung_Opacity-1575.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1575.png,Lung_Opacity/images/Lung_Opacity-1575.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1575.png,Lung_Opacity/masks/Lung_Opacity-1575.png,True,False,299,299,L,104.23,fd19f92a6f8225319d188d32cdb9d9d5
Lung_Opacity-1576.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1576.png,Lung_Opacity/images/Lung_Opacity-1576.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1576.png,Lung_Opacity/masks/Lung_Opacity-1576.png,True,False,299,299,L,101.3,ea01f949c73e257eb39b846a92d9c430
Lung_Opacity-1577.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1577.png,Lung_Opacity/images/Lung_Opacity-1577.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1577.png,Lung_Opacity/masks/Lung_Opacity-1577.png,True,False,299,299,L,114.56,956faed27369aba035f9c0ecbce720cb
Lung_Opacity-1578.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1578.png,Lung_Opacity/images/Lung_Opacity-1578.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1578.png,Lung_Opacity/masks/Lung_Opacity-1578.png,True,False,299,299,L,110.07,49d42ed71e125e6cd970c49cf63eed2a
Lung_Opacity-1579.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1579.png,Lung_Opacity/images/Lung_Opacity-1579.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1579.png,Lung_Opacity/masks/Lung_Opacity-1579.png,True,False,299,299,L,120.12,51765a3dbb54fba6eeb033f2191aae5b
Lung_Opacity-158.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-158.png,Lung_Opacity/images/Lung_Opacity-158.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-158.png,Lung_Opacity/masks/Lung_Opacity-158.png,True,False,299,299,L,108.33,d6a8f31bdbc7b81703fd5da51a850aff
Lung_Opacity-1580.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1580.png,Lung_Opacity/images/Lung_Opacity-1580.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1580.png,Lung_Opacity/masks/Lung_Opacity-1580.png,True,False,299,299,L,135.17,e0e589613998809f847a65bd075807eb
Lung_Opacity-1581.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1581.png,Lung_Opacity/images/Lung_Opacity-1581.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1581.png,Lung_Opacity/masks/Lung_Opacity-1581.png,True,False,299,299,L,149.48,19000f7c516440b0ddcc803ebb33960d
Lung_Opacity-1582.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1582.png,Lung_Opacity/images/Lung_Opacity-1582.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1582.png,Lung_Opacity/masks/Lung_Opacity-1582.png,True,False,299,299,L,137.53,dd217910a5aab8236eec8c9a26075b15
Lung_Opacity-1583.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1583.png,Lung_Opacity/images/Lung_Opacity-1583.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1583.png,Lung_Opacity/masks/Lung_Opacity-1583.png,True,False,299,299,L,140.92,7911db312109bf9ce41b62d79d5a342c
Lung_Opacity-1584.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1584.png,Lung_Opacity/images/Lung_Opacity-1584.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1584.png,Lung_Opacity/masks/Lung_Opacity-1584.png,True,False,299,299,L,114.68,ec3a4a0be99201fa09cd8656473cab87
Lung_Opacity-1585.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1585.png,Lung_Opacity/images/Lung_Opacity-1585.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1585.png,Lung_Opacity/masks/Lung_Opacity-1585.png,True,False,299,299,L,158.62,7abb50ce3db52f63a81fa6ecf5f5c61f
Lung_Opacity-1586.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1586.png,Lung_Opacity/images/Lung_Opacity-1586.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1586.png,Lung_Opacity/masks/Lung_Opacity-1586.png,True,False,299,299,L,122.47,728c137f4e43fc6902df355c1249ec83
Lung_Opacity-1587.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1587.png,Lung_Opacity/images/Lung_Opacity-1587.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1587.png,Lung_Opacity/masks/Lung_Opacity-1587.png,True,False,299,299,L,120.13,0e6cac39cd045ea791b32f80fd12d889
Lung_Opacity-1588.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1588.png,Lung_Opacity/images/Lung_Opacity-1588.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1588.png,Lung_Opacity/masks/Lung_Opacity-1588.png,True,False,299,299,L,152.89,71afac55c883621ae309094cf48d0808
Lung_Opacity-1589.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1589.png,Lung_Opacity/images/Lung_Opacity-1589.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1589.png,Lung_Opacity/masks/Lung_Opacity-1589.png,True,False,299,299,L,107.08,50418c4813ed3f940e8328d3eb452c66
Lung_Opacity-159.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-159.png,Lung_Opacity/images/Lung_Opacity-159.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-159.png,Lung_Opacity/masks/Lung_Opacity-159.png,True,False,299,299,L,157.64,98d33e839ff070ce0e4311771e023d64
Lung_Opacity-1590.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1590.png,Lung_Opacity/images/Lung_Opacity-1590.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1590.png,Lung_Opacity/masks/Lung_Opacity-1590.png,True,False,299,299,L,121.65,8fb3c30fcf4675f4fc617ed3f5896cee
Lung_Opacity-1591.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1591.png,Lung_Opacity/images/Lung_Opacity-1591.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1591.png,Lung_Opacity/masks/Lung_Opacity-1591.png,True,False,299,299,L,111.58,3ba4c5d7e6bed42c2c6afc5f92994bcc
Lung_Opacity-1592.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1592.png,Lung_Opacity/images/Lung_Opacity-1592.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1592.png,Lung_Opacity/masks/Lung_Opacity-1592.png,True,False,299,299,L,112.77,8baa4e58698fad58ba60bc83d1b9e5e0
Lung_Opacity-1593.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1593.png,Lung_Opacity/images/Lung_Opacity-1593.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1593.png,Lung_Opacity/masks/Lung_Opacity-1593.png,True,False,299,299,L,97.1,180656fe4a86355bcbe1c628e680fe15
Lung_Opacity-1594.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1594.png,Lung_Opacity/images/Lung_Opacity-1594.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1594.png,Lung_Opacity/masks/Lung_Opacity-1594.png,True,False,299,299,L,176.97,55f9a76cda43c8e22446b994fb8e618b
Lung_Opacity-1595.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1595.png,Lung_Opacity/images/Lung_Opacity-1595.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1595.png,Lung_Opacity/masks/Lung_Opacity-1595.png,True,False,299,299,L,110.42,9d6c7165efd81a602c9312609fa1f917
Lung_Opacity-1596.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1596.png,Lung_Opacity/images/Lung_Opacity-1596.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1596.png,Lung_Opacity/masks/Lung_Opacity-1596.png,True,False,299,299,L,119.84,e206eb8639aa58f24e1240040d190186
Lung_Opacity-1597.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1597.png,Lung_Opacity/images/Lung_Opacity-1597.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1597.png,Lung_Opacity/masks/Lung_Opacity-1597.png,True,False,299,299,L,106.89,0816d3291cab6986bd5276ec6267ce64
Lung_Opacity-1598.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1598.png,Lung_Opacity/images/Lung_Opacity-1598.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1598.png,Lung_Opacity/masks/Lung_Opacity-1598.png,True,False,299,299,L,97.12,0283fd963f7cdfe1a4f426a3b1b10541
Lung_Opacity-1599.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1599.png,Lung_Opacity/images/Lung_Opacity-1599.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1599.png,Lung_Opacity/masks/Lung_Opacity-1599.png,True,False,299,299,L,137.71,9a288f339d4618d7fd8a35a236892eb8
Lung_Opacity-16.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-16.png,Lung_Opacity/images/Lung_Opacity-16.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-16.png,Lung_Opacity/masks/Lung_Opacity-16.png,True,False,299,299,L,99.49,1ef87004473a9482fa6deaa03288ce4b
Lung_Opacity-160.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-160.png,Lung_Opacity/images/Lung_Opacity-160.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-160.png,Lung_Opacity/masks/Lung_Opacity-160.png,True,False,299,299,L,155.44,75f31331ecdc62f16cc0e18058aeb39b
Lung_Opacity-1600.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1600.png,Lung_Opacity/images/Lung_Opacity-1600.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1600.png,Lung_Opacity/masks/Lung_Opacity-1600.png,True,False,299,299,L,118.5,97c793daa220b62c47037a6d321039c2
Lung_Opacity-1601.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1601.png,Lung_Opacity/images/Lung_Opacity-1601.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1601.png,Lung_Opacity/masks/Lung_Opacity-1601.png,True,False,299,299,L,115.43,bd488b07f1414836273d4ef4f0d0a0fb
Lung_Opacity-1602.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1602.png,Lung_Opacity/images/Lung_Opacity-1602.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1602.png,Lung_Opacity/masks/Lung_Opacity-1602.png,True,False,299,299,L,110.46,372a6cc225ed5d60c936bf4ea8c6b048
Lung_Opacity-1603.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1603.png,Lung_Opacity/images/Lung_Opacity-1603.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1603.png,Lung_Opacity/masks/Lung_Opacity-1603.png,True,False,299,299,L,107.28,b19743fe2c6ff2b3f21d974246029c3f
Lung_Opacity-1604.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1604.png,Lung_Opacity/images/Lung_Opacity-1604.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1604.png,Lung_Opacity/masks/Lung_Opacity-1604.png,True,False,299,299,L,168.94,bd433d06132fc0cbbd24e6f8a317f44d
Lung_Opacity-1605.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1605.png,Lung_Opacity/images/Lung_Opacity-1605.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1605.png,Lung_Opacity/masks/Lung_Opacity-1605.png,True,False,299,299,L,105.88,53d4ab78b82463a28215407dae70596b
Lung_Opacity-1606.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1606.png,Lung_Opacity/images/Lung_Opacity-1606.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1606.png,Lung_Opacity/masks/Lung_Opacity-1606.png,True,False,299,299,L,114.34,ffca5478ce4c1667b788c8bb7925a4fe
Lung_Opacity-1607.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1607.png,Lung_Opacity/images/Lung_Opacity-1607.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1607.png,Lung_Opacity/masks/Lung_Opacity-1607.png,True,False,299,299,L,111.03,94d61c61bceea6cd17ef3fe69424cc9d
Lung_Opacity-1608.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1608.png,Lung_Opacity/images/Lung_Opacity-1608.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1608.png,Lung_Opacity/masks/Lung_Opacity-1608.png,True,False,299,299,L,165.33,e0385ecfc9ea169290f94edcb4e18fab
Lung_Opacity-1609.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1609.png,Lung_Opacity/images/Lung_Opacity-1609.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1609.png,Lung_Opacity/masks/Lung_Opacity-1609.png,True,False,299,299,L,119.89,3ab41d29b24baf33850722fcd9a31643
Lung_Opacity-161.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-161.png,Lung_Opacity/images/Lung_Opacity-161.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-161.png,Lung_Opacity/masks/Lung_Opacity-161.png,True,False,299,299,L,164.86,37e1282cc299d655e56547ef908d72b4
Lung_Opacity-1610.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1610.png,Lung_Opacity/images/Lung_Opacity-1610.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1610.png,Lung_Opacity/masks/Lung_Opacity-1610.png,True,False,299,299,L,118.47,e4a0bc36049dfc1b3bd8f20c75bafd7b
Lung_Opacity-1611.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1611.png,Lung_Opacity/images/Lung_Opacity-1611.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1611.png,Lung_Opacity/masks/Lung_Opacity-1611.png,True,False,299,299,L,120.95,ae55bffe4aadaae0050b96d0bcd95eab
Lung_Opacity-1612.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1612.png,Lung_Opacity/images/Lung_Opacity-1612.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1612.png,Lung_Opacity/masks/Lung_Opacity-1612.png,True,False,299,299,L,172.5,18a88d8dcd3c942d91b5c418132c2615
Lung_Opacity-1613.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1613.png,Lung_Opacity/images/Lung_Opacity-1613.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1613.png,Lung_Opacity/masks/Lung_Opacity-1613.png,True,False,299,299,L,119.03,91407f414ae20b081c1b426bc4713816
Lung_Opacity-1614.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1614.png,Lung_Opacity/images/Lung_Opacity-1614.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1614.png,Lung_Opacity/masks/Lung_Opacity-1614.png,True,False,299,299,L,130.11,e493be50fe967beb65268b365bead350
Lung_Opacity-1615.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1615.png,Lung_Opacity/images/Lung_Opacity-1615.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1615.png,Lung_Opacity/masks/Lung_Opacity-1615.png,True,False,299,299,L,118.98,26d473c96b59945376c3d89595ce152c
Lung_Opacity-1616.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1616.png,Lung_Opacity/images/Lung_Opacity-1616.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1616.png,Lung_Opacity/masks/Lung_Opacity-1616.png,True,False,299,299,L,137.8,1ccfbb8e5a549f935a32a224617f7cb1
Lung_Opacity-1617.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1617.png,Lung_Opacity/images/Lung_Opacity-1617.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1617.png,Lung_Opacity/masks/Lung_Opacity-1617.png,True,False,299,299,L,150.24,ccd5a4903c04fc962286547398a08a90
Lung_Opacity-1618.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1618.png,Lung_Opacity/images/Lung_Opacity-1618.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1618.png,Lung_Opacity/masks/Lung_Opacity-1618.png,True,False,299,299,L,111.69,3db8d77e7842d840cdda621bbe6f51e0
Lung_Opacity-1619.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1619.png,Lung_Opacity/images/Lung_Opacity-1619.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1619.png,Lung_Opacity/masks/Lung_Opacity-1619.png,True,False,299,299,L,104.57,b8e83ca37727c7a08b9807c7a51f2dd9
Lung_Opacity-162.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-162.png,Lung_Opacity/images/Lung_Opacity-162.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-162.png,Lung_Opacity/masks/Lung_Opacity-162.png,True,False,299,299,L,85.47,243d2e831aba176cb008392d93fcb48b
Lung_Opacity-1620.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1620.png,Lung_Opacity/images/Lung_Opacity-1620.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1620.png,Lung_Opacity/masks/Lung_Opacity-1620.png,True,False,299,299,L,170.28,320480941793c430cfa61d3c3d523ba2
Lung_Opacity-1621.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1621.png,Lung_Opacity/images/Lung_Opacity-1621.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1621.png,Lung_Opacity/masks/Lung_Opacity-1621.png,True,False,299,299,L,181.84,092a45336bddc3ddccde96cbc63d1dad
Lung_Opacity-1622.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1622.png,Lung_Opacity/images/Lung_Opacity-1622.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1622.png,Lung_Opacity/masks/Lung_Opacity-1622.png,True,False,299,299,L,122.0,5bbff8c37a7d333dde28b6b6ddd95b85
Lung_Opacity-1623.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1623.png,Lung_Opacity/images/Lung_Opacity-1623.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1623.png,Lung_Opacity/masks/Lung_Opacity-1623.png,True,False,299,299,L,118.02,ea5549bf4fc34bdd0a9da5fdef57dc4b
Lung_Opacity-1624.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1624.png,Lung_Opacity/images/Lung_Opacity-1624.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1624.png,Lung_Opacity/masks/Lung_Opacity-1624.png,True,False,299,299,L,114.97,a60935fd3e41ac3a7dfc41bd0178dff3
Lung_Opacity-1625.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1625.png,Lung_Opacity/images/Lung_Opacity-1625.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1625.png,Lung_Opacity/masks/Lung_Opacity-1625.png,True,False,299,299,L,105.49,98e3b72706a86d7e577212950d4d8f5c
Lung_Opacity-1626.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1626.png,Lung_Opacity/images/Lung_Opacity-1626.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1626.png,Lung_Opacity/masks/Lung_Opacity-1626.png,True,False,299,299,L,149.84,aec180aa4ce38282168c6764e8de85da
Lung_Opacity-1627.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1627.png,Lung_Opacity/images/Lung_Opacity-1627.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1627.png,Lung_Opacity/masks/Lung_Opacity-1627.png,True,False,299,299,L,116.71,58ec6b279cc68215968e7d851c085216
Lung_Opacity-1628.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1628.png,Lung_Opacity/images/Lung_Opacity-1628.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1628.png,Lung_Opacity/masks/Lung_Opacity-1628.png,True,False,299,299,L,118.18,86bd05e502c12b569f26cc69ec803bab
Lung_Opacity-1629.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1629.png,Lung_Opacity/images/Lung_Opacity-1629.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1629.png,Lung_Opacity/masks/Lung_Opacity-1629.png,True,False,299,299,L,110.03,6711d5801004d54c1b5b066e6290cb1a
Lung_Opacity-163.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-163.png,Lung_Opacity/images/Lung_Opacity-163.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-163.png,Lung_Opacity/masks/Lung_Opacity-163.png,True,False,299,299,L,128.39,004265872c3915c7094bf0ab2ca7d3bb
Lung_Opacity-1630.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1630.png,Lung_Opacity/images/Lung_Opacity-1630.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1630.png,Lung_Opacity/masks/Lung_Opacity-1630.png,True,False,299,299,L,102.31,a8e3594e1e99b1d64254e5a5a269feaa
Lung_Opacity-1631.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1631.png,Lung_Opacity/images/Lung_Opacity-1631.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1631.png,Lung_Opacity/masks/Lung_Opacity-1631.png,True,False,299,299,L,154.86,f14922e222b6efc21d08d845b195c01f
Lung_Opacity-1632.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1632.png,Lung_Opacity/images/Lung_Opacity-1632.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1632.png,Lung_Opacity/masks/Lung_Opacity-1632.png,True,False,299,299,L,158.52,90cd8ef403b6ef01f9222830c938d620
Lung_Opacity-1633.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1633.png,Lung_Opacity/images/Lung_Opacity-1633.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1633.png,Lung_Opacity/masks/Lung_Opacity-1633.png,True,False,299,299,L,152.98,53ca5a8b12855171d8cbb8cb2f41e97e
Lung_Opacity-1634.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1634.png,Lung_Opacity/images/Lung_Opacity-1634.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1634.png,Lung_Opacity/masks/Lung_Opacity-1634.png,True,False,299,299,L,97.04,f4e8e234f1f7e6c8176843b7a69ced20
Lung_Opacity-1635.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1635.png,Lung_Opacity/images/Lung_Opacity-1635.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1635.png,Lung_Opacity/masks/Lung_Opacity-1635.png,True,False,299,299,L,143.7,a7c5823becb3096c1ae3014a435e11fa
Lung_Opacity-1636.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1636.png,Lung_Opacity/images/Lung_Opacity-1636.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1636.png,Lung_Opacity/masks/Lung_Opacity-1636.png,True,False,299,299,L,174.76,0cf9c93a2cd39e5c2ba7dd9529cb0afd
Lung_Opacity-1637.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1637.png,Lung_Opacity/images/Lung_Opacity-1637.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1637.png,Lung_Opacity/masks/Lung_Opacity-1637.png,True,False,299,299,L,159.84,7034a5334fa0c6655ea564c8b2dc03a5
Lung_Opacity-1638.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1638.png,Lung_Opacity/images/Lung_Opacity-1638.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1638.png,Lung_Opacity/masks/Lung_Opacity-1638.png,True,False,299,299,L,158.73,7abcba12fe66f9aac9af63ddbfbb63fd
Lung_Opacity-1639.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1639.png,Lung_Opacity/images/Lung_Opacity-1639.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1639.png,Lung_Opacity/masks/Lung_Opacity-1639.png,True,False,299,299,L,103.72,43c287874cb1c44d89eb4b64419e46fb
Lung_Opacity-164.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-164.png,Lung_Opacity/images/Lung_Opacity-164.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-164.png,Lung_Opacity/masks/Lung_Opacity-164.png,True,False,299,299,L,117.49,0ffc9933f1934191f844195e7a53c269
Lung_Opacity-1640.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1640.png,Lung_Opacity/images/Lung_Opacity-1640.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1640.png,Lung_Opacity/masks/Lung_Opacity-1640.png,True,False,299,299,L,103.47,88b1b00ce10a7d361f3a195704139528
Lung_Opacity-1641.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1641.png,Lung_Opacity/images/Lung_Opacity-1641.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1641.png,Lung_Opacity/masks/Lung_Opacity-1641.png,True,False,299,299,L,129.52,6cd9094c5a99f7d59d26c28fdc74be80
Lung_Opacity-1642.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1642.png,Lung_Opacity/images/Lung_Opacity-1642.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1642.png,Lung_Opacity/masks/Lung_Opacity-1642.png,True,False,299,299,L,95.0,e34b671cfe65f3c9dea20fcaf47416a7
Lung_Opacity-1643.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1643.png,Lung_Opacity/images/Lung_Opacity-1643.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1643.png,Lung_Opacity/masks/Lung_Opacity-1643.png,True,False,299,299,L,98.32,e26177ec9337ef8940f56b8ffd129115
Lung_Opacity-1644.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1644.png,Lung_Opacity/images/Lung_Opacity-1644.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1644.png,Lung_Opacity/masks/Lung_Opacity-1644.png,True,False,299,299,L,114.5,49823b339253134a0d65a8e7b3819932
Lung_Opacity-1645.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1645.png,Lung_Opacity/images/Lung_Opacity-1645.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1645.png,Lung_Opacity/masks/Lung_Opacity-1645.png,True,False,299,299,L,116.52,d94ad174e07073dd105f3c8133e1709e
Lung_Opacity-1646.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1646.png,Lung_Opacity/images/Lung_Opacity-1646.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1646.png,Lung_Opacity/masks/Lung_Opacity-1646.png,True,False,299,299,L,110.31,ac26e7e87669fb9310ab5fb5fc67a059
Lung_Opacity-1647.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1647.png,Lung_Opacity/images/Lung_Opacity-1647.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1647.png,Lung_Opacity/masks/Lung_Opacity-1647.png,True,False,299,299,L,111.96,5bfcc78ad8a178cb1a355d121df934a2
Lung_Opacity-1648.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1648.png,Lung_Opacity/images/Lung_Opacity-1648.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1648.png,Lung_Opacity/masks/Lung_Opacity-1648.png,True,False,299,299,L,119.06,18cba62f2b84c27beec0457a0d9c7905
Lung_Opacity-1649.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1649.png,Lung_Opacity/images/Lung_Opacity-1649.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1649.png,Lung_Opacity/masks/Lung_Opacity-1649.png,True,False,299,299,L,133.88,7101102a363575d7a5ee05ab54557419
Lung_Opacity-165.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-165.png,Lung_Opacity/images/Lung_Opacity-165.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-165.png,Lung_Opacity/masks/Lung_Opacity-165.png,True,False,299,299,L,118.49,4911130de5442e5e7531ddfeadb05439
Lung_Opacity-1650.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1650.png,Lung_Opacity/images/Lung_Opacity-1650.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1650.png,Lung_Opacity/masks/Lung_Opacity-1650.png,True,False,299,299,L,115.45,1559c004643f3a576aa06be53ca018b2
Lung_Opacity-1651.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1651.png,Lung_Opacity/images/Lung_Opacity-1651.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1651.png,Lung_Opacity/masks/Lung_Opacity-1651.png,True,False,299,299,L,124.11,dcc81cd04ee8724b0a6f2b86d889b182
Lung_Opacity-1652.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1652.png,Lung_Opacity/images/Lung_Opacity-1652.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1652.png,Lung_Opacity/masks/Lung_Opacity-1652.png,True,False,299,299,L,105.71,a9124b6b490faad8b7fab150f355c335
Lung_Opacity-1653.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1653.png,Lung_Opacity/images/Lung_Opacity-1653.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1653.png,Lung_Opacity/masks/Lung_Opacity-1653.png,True,False,299,299,L,134.88,76ee34535b6cd35f16e0aadf897ff582
Lung_Opacity-1654.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1654.png,Lung_Opacity/images/Lung_Opacity-1654.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1654.png,Lung_Opacity/masks/Lung_Opacity-1654.png,True,False,299,299,L,102.7,cec5e50f8076e215d4269e3ff77add29
Lung_Opacity-1655.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1655.png,Lung_Opacity/images/Lung_Opacity-1655.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1655.png,Lung_Opacity/masks/Lung_Opacity-1655.png,True,False,299,299,L,116.56,1dba37e49f726142f0db93fcc6c7910c
Lung_Opacity-1656.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1656.png,Lung_Opacity/images/Lung_Opacity-1656.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1656.png,Lung_Opacity/masks/Lung_Opacity-1656.png,True,False,299,299,L,114.71,dd85270685d0d70e074af5f96e228543
Lung_Opacity-1657.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1657.png,Lung_Opacity/images/Lung_Opacity-1657.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1657.png,Lung_Opacity/masks/Lung_Opacity-1657.png,True,False,299,299,L,111.45,4100a71cc2d1cab0de957cb44c8a7609
Lung_Opacity-1658.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1658.png,Lung_Opacity/images/Lung_Opacity-1658.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1658.png,Lung_Opacity/masks/Lung_Opacity-1658.png,True,False,299,299,L,100.24,020fa724f3d488a1b70d199d519583fc
Lung_Opacity-1659.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1659.png,Lung_Opacity/images/Lung_Opacity-1659.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1659.png,Lung_Opacity/masks/Lung_Opacity-1659.png,True,False,299,299,L,121.11,b346ce2b0d682f6147d036cb85d2a7fc
Lung_Opacity-166.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-166.png,Lung_Opacity/images/Lung_Opacity-166.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-166.png,Lung_Opacity/masks/Lung_Opacity-166.png,True,False,299,299,L,139.58,f1d28e321e81925bc47d2dbc0139e55d
Lung_Opacity-1660.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1660.png,Lung_Opacity/images/Lung_Opacity-1660.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1660.png,Lung_Opacity/masks/Lung_Opacity-1660.png,True,False,299,299,L,159.81,44ab23043cd35b242ecbfb6950607c61
Lung_Opacity-1661.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1661.png,Lung_Opacity/images/Lung_Opacity-1661.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1661.png,Lung_Opacity/masks/Lung_Opacity-1661.png,True,False,299,299,L,114.14,534c26b45841911ca9cb65d33e33a00b
Lung_Opacity-1662.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1662.png,Lung_Opacity/images/Lung_Opacity-1662.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1662.png,Lung_Opacity/masks/Lung_Opacity-1662.png,True,False,299,299,L,96.52,8defb2b8d4ada8098d673457a5f8391c
Lung_Opacity-1663.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1663.png,Lung_Opacity/images/Lung_Opacity-1663.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1663.png,Lung_Opacity/masks/Lung_Opacity-1663.png,True,False,299,299,L,95.28,0bb985ce8c0ffd0dd0d3cf0ee1f91657
Lung_Opacity-1664.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1664.png,Lung_Opacity/images/Lung_Opacity-1664.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1664.png,Lung_Opacity/masks/Lung_Opacity-1664.png,True,False,299,299,L,116.55,8cc4b2554eadea757e3a5b1a6b8689c1
Lung_Opacity-1665.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1665.png,Lung_Opacity/images/Lung_Opacity-1665.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1665.png,Lung_Opacity/masks/Lung_Opacity-1665.png,True,False,299,299,L,126.3,5ca98db0af3e83b37411303c5a85ddc0
Lung_Opacity-1666.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1666.png,Lung_Opacity/images/Lung_Opacity-1666.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1666.png,Lung_Opacity/masks/Lung_Opacity-1666.png,True,False,299,299,L,156.21,de6549d56ba1b8606fb2f0aa0062a51c
Lung_Opacity-1667.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1667.png,Lung_Opacity/images/Lung_Opacity-1667.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1667.png,Lung_Opacity/masks/Lung_Opacity-1667.png,True,False,299,299,L,113.54,5276c98a65edfbf51b66ad3342c8c9d3
Lung_Opacity-1668.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1668.png,Lung_Opacity/images/Lung_Opacity-1668.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1668.png,Lung_Opacity/masks/Lung_Opacity-1668.png,True,False,299,299,L,125.75,21f299d29ad66a66a2de48d8272765a0
Lung_Opacity-1669.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1669.png,Lung_Opacity/images/Lung_Opacity-1669.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1669.png,Lung_Opacity/masks/Lung_Opacity-1669.png,True,False,299,299,L,124.59,9768787660df7e18d8a2a15554fac495
Lung_Opacity-167.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-167.png,Lung_Opacity/images/Lung_Opacity-167.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-167.png,Lung_Opacity/masks/Lung_Opacity-167.png,True,False,299,299,L,151.22,13c1a5f231a333bd71279d8f819611b4
Lung_Opacity-1670.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1670.png,Lung_Opacity/images/Lung_Opacity-1670.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1670.png,Lung_Opacity/masks/Lung_Opacity-1670.png,True,False,299,299,L,131.95,017d6da3b9ae463c1299a87fa3085b49
Lung_Opacity-1671.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1671.png,Lung_Opacity/images/Lung_Opacity-1671.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1671.png,Lung_Opacity/masks/Lung_Opacity-1671.png,True,False,299,299,L,160.88,e856556b81354f1014110e6e27202796
Lung_Opacity-1672.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1672.png,Lung_Opacity/images/Lung_Opacity-1672.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1672.png,Lung_Opacity/masks/Lung_Opacity-1672.png,True,False,299,299,L,73.36,3c5b1cd351766cdd958d23e87558b1d6
Lung_Opacity-1673.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1673.png,Lung_Opacity/images/Lung_Opacity-1673.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1673.png,Lung_Opacity/masks/Lung_Opacity-1673.png,True,False,299,299,L,134.76,3ba9c9b869574195e0a702a78da00f21
Lung_Opacity-1674.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1674.png,Lung_Opacity/images/Lung_Opacity-1674.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1674.png,Lung_Opacity/masks/Lung_Opacity-1674.png,True,False,299,299,L,135.69,c746929134715b94e4fa18d9d3df54cf
Lung_Opacity-1675.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1675.png,Lung_Opacity/images/Lung_Opacity-1675.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1675.png,Lung_Opacity/masks/Lung_Opacity-1675.png,True,False,299,299,L,152.04,5fc2b9737ad24133ffbb972de8885269
Lung_Opacity-1676.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1676.png,Lung_Opacity/images/Lung_Opacity-1676.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1676.png,Lung_Opacity/masks/Lung_Opacity-1676.png,True,False,299,299,L,138.61,4925e39a0b39584375b063398ceb6b73
Lung_Opacity-1677.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1677.png,Lung_Opacity/images/Lung_Opacity-1677.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1677.png,Lung_Opacity/masks/Lung_Opacity-1677.png,True,False,299,299,L,159.84,43abd8eb75e07bc747f2636c30b4d3e9
Lung_Opacity-1678.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1678.png,Lung_Opacity/images/Lung_Opacity-1678.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1678.png,Lung_Opacity/masks/Lung_Opacity-1678.png,True,False,299,299,L,106.2,359dd852543ea6e0638cd75f3f727d20
Lung_Opacity-1679.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1679.png,Lung_Opacity/images/Lung_Opacity-1679.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1679.png,Lung_Opacity/masks/Lung_Opacity-1679.png,True,False,299,299,L,112.86,3cc44c9306fbf129fb1db2efb8a257b9
Lung_Opacity-168.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-168.png,Lung_Opacity/images/Lung_Opacity-168.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-168.png,Lung_Opacity/masks/Lung_Opacity-168.png,True,False,299,299,L,106.94,3e85b7093685592f0cde393f87d4d343
Lung_Opacity-1680.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1680.png,Lung_Opacity/images/Lung_Opacity-1680.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1680.png,Lung_Opacity/masks/Lung_Opacity-1680.png,True,False,299,299,L,103.09,5e7d1723a779ebc93cbe9416d1cd06be
Lung_Opacity-1681.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1681.png,Lung_Opacity/images/Lung_Opacity-1681.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1681.png,Lung_Opacity/masks/Lung_Opacity-1681.png,True,False,299,299,L,106.87,e31267edc159b1b50bf79024dd14d40f
Lung_Opacity-1682.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1682.png,Lung_Opacity/images/Lung_Opacity-1682.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1682.png,Lung_Opacity/masks/Lung_Opacity-1682.png,True,False,299,299,L,121.51,567c81b6dfcd55ed6b3f4b490218094b
Lung_Opacity-1683.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1683.png,Lung_Opacity/images/Lung_Opacity-1683.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1683.png,Lung_Opacity/masks/Lung_Opacity-1683.png,True,False,299,299,L,119.53,843530abfeadae3638232bf7ef84fe45
Lung_Opacity-1684.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1684.png,Lung_Opacity/images/Lung_Opacity-1684.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1684.png,Lung_Opacity/masks/Lung_Opacity-1684.png,True,False,299,299,L,139.7,05d5c65da515fdb0c85037181e62a4c3
Lung_Opacity-1685.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1685.png,Lung_Opacity/images/Lung_Opacity-1685.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1685.png,Lung_Opacity/masks/Lung_Opacity-1685.png,True,False,299,299,L,131.12,e56f645afdb5838bbb322d58e6e9b552
Lung_Opacity-1686.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1686.png,Lung_Opacity/images/Lung_Opacity-1686.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1686.png,Lung_Opacity/masks/Lung_Opacity-1686.png,True,False,299,299,L,136.95,76d27aed44558b41d6ce3000410913df
Lung_Opacity-1687.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1687.png,Lung_Opacity/images/Lung_Opacity-1687.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1687.png,Lung_Opacity/masks/Lung_Opacity-1687.png,True,False,299,299,L,122.72,2e6fbdb1140d1bead0298436b5ae7781
Lung_Opacity-1688.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1688.png,Lung_Opacity/images/Lung_Opacity-1688.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1688.png,Lung_Opacity/masks/Lung_Opacity-1688.png,True,False,299,299,L,115.13,a20ab1749a076e63d491dbbc84b7ce18
Lung_Opacity-1689.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1689.png,Lung_Opacity/images/Lung_Opacity-1689.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1689.png,Lung_Opacity/masks/Lung_Opacity-1689.png,True,False,299,299,L,170.79,6f722d2b7c7b38a038bc8cd015cdcc09
Lung_Opacity-169.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-169.png,Lung_Opacity/images/Lung_Opacity-169.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-169.png,Lung_Opacity/masks/Lung_Opacity-169.png,True,False,299,299,L,112.6,bb3b3a1b33c0b244519960c9b58e7586
Lung_Opacity-1690.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1690.png,Lung_Opacity/images/Lung_Opacity-1690.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1690.png,Lung_Opacity/masks/Lung_Opacity-1690.png,True,False,299,299,L,107.27,4b56c46ad8b6d366956da7ce7d8b78b6
Lung_Opacity-1691.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1691.png,Lung_Opacity/images/Lung_Opacity-1691.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1691.png,Lung_Opacity/masks/Lung_Opacity-1691.png,True,False,299,299,L,119.08,f971d174b83459fb239c484390b05f3c
Lung_Opacity-1692.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1692.png,Lung_Opacity/images/Lung_Opacity-1692.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1692.png,Lung_Opacity/masks/Lung_Opacity-1692.png,True,False,299,299,L,170.65,3d837b6cd2d638464e3897608bb4015a
Lung_Opacity-1693.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1693.png,Lung_Opacity/images/Lung_Opacity-1693.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1693.png,Lung_Opacity/masks/Lung_Opacity-1693.png,True,False,299,299,L,174.42,abfd16b7d801147fcf18c26cad92900e
Lung_Opacity-1694.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1694.png,Lung_Opacity/images/Lung_Opacity-1694.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1694.png,Lung_Opacity/masks/Lung_Opacity-1694.png,True,False,299,299,L,121.59,ee6201a9994f604322645c95d07f0e09
Lung_Opacity-1695.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1695.png,Lung_Opacity/images/Lung_Opacity-1695.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1695.png,Lung_Opacity/masks/Lung_Opacity-1695.png,True,False,299,299,L,99.25,d5dec434442333d5173baedb2914bb6e
Lung_Opacity-1696.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1696.png,Lung_Opacity/images/Lung_Opacity-1696.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1696.png,Lung_Opacity/masks/Lung_Opacity-1696.png,True,False,299,299,L,167.0,4274359121ce114075b03078daf2b143
Lung_Opacity-1697.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1697.png,Lung_Opacity/images/Lung_Opacity-1697.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1697.png,Lung_Opacity/masks/Lung_Opacity-1697.png,True,False,299,299,L,133.41,79b7378455c47b5639e82f794c5af9b4
Lung_Opacity-1698.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1698.png,Lung_Opacity/images/Lung_Opacity-1698.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1698.png,Lung_Opacity/masks/Lung_Opacity-1698.png,True,False,299,299,L,121.22,07901b186f403e0a45801bfbb72813eb
Lung_Opacity-1699.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1699.png,Lung_Opacity/images/Lung_Opacity-1699.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1699.png,Lung_Opacity/masks/Lung_Opacity-1699.png,True,False,299,299,L,136.47,c000b7a35d509b5574249817420b39ee
Lung_Opacity-17.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-17.png,Lung_Opacity/images/Lung_Opacity-17.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-17.png,Lung_Opacity/masks/Lung_Opacity-17.png,True,False,299,299,L,175.57,3400cb0465e3a8fbe2bb8468309e30a6
Lung_Opacity-170.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-170.png,Lung_Opacity/images/Lung_Opacity-170.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-170.png,Lung_Opacity/masks/Lung_Opacity-170.png,True,False,299,299,L,135.34,bc1fae5c914a5c6f0932c36a4677537f
Lung_Opacity-1700.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1700.png,Lung_Opacity/images/Lung_Opacity-1700.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1700.png,Lung_Opacity/masks/Lung_Opacity-1700.png,True,False,299,299,L,117.03,985d9bbcf900230eab8168fd99599d31
Lung_Opacity-1701.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1701.png,Lung_Opacity/images/Lung_Opacity-1701.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1701.png,Lung_Opacity/masks/Lung_Opacity-1701.png,True,False,299,299,L,120.1,e2b5ef7dd515495258a22fb097a6c464
Lung_Opacity-1702.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1702.png,Lung_Opacity/images/Lung_Opacity-1702.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1702.png,Lung_Opacity/masks/Lung_Opacity-1702.png,True,False,299,299,L,103.26,76556b4099b3628bddf85a3c8e813799
Lung_Opacity-1703.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1703.png,Lung_Opacity/images/Lung_Opacity-1703.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1703.png,Lung_Opacity/masks/Lung_Opacity-1703.png,True,False,299,299,L,162.48,e45170b540ca7359c03155e41e2bac86
Lung_Opacity-1704.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1704.png,Lung_Opacity/images/Lung_Opacity-1704.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1704.png,Lung_Opacity/masks/Lung_Opacity-1704.png,True,False,299,299,L,143.96,9dc33c322fd033d3b5c2c247826253c8
Lung_Opacity-1705.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1705.png,Lung_Opacity/images/Lung_Opacity-1705.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1705.png,Lung_Opacity/masks/Lung_Opacity-1705.png,True,False,299,299,L,163.42,e1335cc3435dd7e576c7506378930906
Lung_Opacity-1706.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1706.png,Lung_Opacity/images/Lung_Opacity-1706.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1706.png,Lung_Opacity/masks/Lung_Opacity-1706.png,True,False,299,299,L,118.17,4a1c7289a96e1c34b6ad681dc070e955
Lung_Opacity-1707.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1707.png,Lung_Opacity/images/Lung_Opacity-1707.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1707.png,Lung_Opacity/masks/Lung_Opacity-1707.png,True,False,299,299,L,86.42,1612a698b398c77a7feb13c4ebe9875e
Lung_Opacity-1708.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1708.png,Lung_Opacity/images/Lung_Opacity-1708.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1708.png,Lung_Opacity/masks/Lung_Opacity-1708.png,True,False,299,299,L,120.06,9f2ab465085d8026648798e121c4f62f
Lung_Opacity-1709.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1709.png,Lung_Opacity/images/Lung_Opacity-1709.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1709.png,Lung_Opacity/masks/Lung_Opacity-1709.png,True,False,299,299,L,105.62,33aa5e06e638ed735b84dfd8ef1443d1
Lung_Opacity-171.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-171.png,Lung_Opacity/images/Lung_Opacity-171.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-171.png,Lung_Opacity/masks/Lung_Opacity-171.png,True,False,299,299,L,157.02,425799a794b45eb50b80e034fbae7c91
Lung_Opacity-1710.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1710.png,Lung_Opacity/images/Lung_Opacity-1710.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1710.png,Lung_Opacity/masks/Lung_Opacity-1710.png,True,False,299,299,L,161.03,c48330c63e423ec32ce8a9039d43dc2c
Lung_Opacity-1711.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1711.png,Lung_Opacity/images/Lung_Opacity-1711.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1711.png,Lung_Opacity/masks/Lung_Opacity-1711.png,True,False,299,299,L,112.76,ce570ea4ed05e9f4349988d7e25c3800
Lung_Opacity-1712.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1712.png,Lung_Opacity/images/Lung_Opacity-1712.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1712.png,Lung_Opacity/masks/Lung_Opacity-1712.png,True,False,299,299,L,148.7,3149983391a6b1f2deeb2ac2302242a9
Lung_Opacity-1713.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1713.png,Lung_Opacity/images/Lung_Opacity-1713.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1713.png,Lung_Opacity/masks/Lung_Opacity-1713.png,True,False,299,299,L,106.61,d7550cbad05a3cd8be4bf7fe3d4b99c7
Lung_Opacity-1714.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1714.png,Lung_Opacity/images/Lung_Opacity-1714.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1714.png,Lung_Opacity/masks/Lung_Opacity-1714.png,True,False,299,299,L,115.51,f899c4d0720c1972ec67e9d0199a9c4d
Lung_Opacity-1715.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1715.png,Lung_Opacity/images/Lung_Opacity-1715.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1715.png,Lung_Opacity/masks/Lung_Opacity-1715.png,True,False,299,299,L,129.55,3fbcd059aea71799c9e9994947894d29
Lung_Opacity-1716.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1716.png,Lung_Opacity/images/Lung_Opacity-1716.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1716.png,Lung_Opacity/masks/Lung_Opacity-1716.png,True,False,299,299,L,154.83,8ac01cbe0c28b965933473d3a13003d1
Lung_Opacity-1717.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1717.png,Lung_Opacity/images/Lung_Opacity-1717.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1717.png,Lung_Opacity/masks/Lung_Opacity-1717.png,True,False,299,299,L,140.29,d7fa1cea03c1bddcdbfdf718edc27220
Lung_Opacity-1718.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1718.png,Lung_Opacity/images/Lung_Opacity-1718.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1718.png,Lung_Opacity/masks/Lung_Opacity-1718.png,True,False,299,299,L,157.81,3b9cd39b7736b166118294e069417041
Lung_Opacity-1719.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1719.png,Lung_Opacity/images/Lung_Opacity-1719.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1719.png,Lung_Opacity/masks/Lung_Opacity-1719.png,True,False,299,299,L,110.4,f2cb579514206d1f0466703b70794a30
Lung_Opacity-172.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-172.png,Lung_Opacity/images/Lung_Opacity-172.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-172.png,Lung_Opacity/masks/Lung_Opacity-172.png,True,False,299,299,L,130.13,fb7e4b692274ca285674f2fe9bc9e33f
Lung_Opacity-1720.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1720.png,Lung_Opacity/images/Lung_Opacity-1720.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1720.png,Lung_Opacity/masks/Lung_Opacity-1720.png,True,False,299,299,L,116.51,d2ee1cfd8c74f71cbfcfd4f7d28b0e03
Lung_Opacity-1721.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1721.png,Lung_Opacity/images/Lung_Opacity-1721.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1721.png,Lung_Opacity/masks/Lung_Opacity-1721.png,True,False,299,299,L,109.21,ab40a7168975db6d52c35be71379c6e6
Lung_Opacity-1722.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1722.png,Lung_Opacity/images/Lung_Opacity-1722.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1722.png,Lung_Opacity/masks/Lung_Opacity-1722.png,True,False,299,299,L,107.47,21b851c57e9b996a7e894cf92baf7607
Lung_Opacity-1723.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1723.png,Lung_Opacity/images/Lung_Opacity-1723.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1723.png,Lung_Opacity/masks/Lung_Opacity-1723.png,True,False,299,299,L,105.24,6c3c25bc87e6e2fcb84f071d67b154a7
Lung_Opacity-1724.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1724.png,Lung_Opacity/images/Lung_Opacity-1724.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1724.png,Lung_Opacity/masks/Lung_Opacity-1724.png,True,False,299,299,L,130.29,37d72c8321d94e3052bc962f2461b900
Lung_Opacity-1725.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1725.png,Lung_Opacity/images/Lung_Opacity-1725.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1725.png,Lung_Opacity/masks/Lung_Opacity-1725.png,True,False,299,299,L,112.34,52f7f5618ae475b2208029ab687a1119
Lung_Opacity-1726.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1726.png,Lung_Opacity/images/Lung_Opacity-1726.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1726.png,Lung_Opacity/masks/Lung_Opacity-1726.png,True,False,299,299,L,120.45,58ef60613fd09f96e5cf2ba824db9097
Lung_Opacity-1727.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1727.png,Lung_Opacity/images/Lung_Opacity-1727.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1727.png,Lung_Opacity/masks/Lung_Opacity-1727.png,True,False,299,299,L,108.0,eb2066b948cd41462db1a6cd1fbdabb3
Lung_Opacity-1728.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1728.png,Lung_Opacity/images/Lung_Opacity-1728.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1728.png,Lung_Opacity/masks/Lung_Opacity-1728.png,True,False,299,299,L,104.39,f05d2d7de7afd680abb639be993d66ee
Lung_Opacity-1729.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1729.png,Lung_Opacity/images/Lung_Opacity-1729.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1729.png,Lung_Opacity/masks/Lung_Opacity-1729.png,True,False,299,299,L,106.74,c7c2091ee7b91b04aa3ed8965a0c609f
Lung_Opacity-173.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-173.png,Lung_Opacity/images/Lung_Opacity-173.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-173.png,Lung_Opacity/masks/Lung_Opacity-173.png,True,False,299,299,L,165.11,c6ca90e1abd83ca5e1c2c55d38f75567
Lung_Opacity-1730.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1730.png,Lung_Opacity/images/Lung_Opacity-1730.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1730.png,Lung_Opacity/masks/Lung_Opacity-1730.png,True,False,299,299,L,136.88,f5bdc53b88089a0b67b7b68076865c1d
Lung_Opacity-1731.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1731.png,Lung_Opacity/images/Lung_Opacity-1731.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1731.png,Lung_Opacity/masks/Lung_Opacity-1731.png,True,False,299,299,L,115.45,c881024eb479c0cf411e73a1e19dc111
Lung_Opacity-1732.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1732.png,Lung_Opacity/images/Lung_Opacity-1732.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1732.png,Lung_Opacity/masks/Lung_Opacity-1732.png,True,False,299,299,L,112.53,30ff2d6bb5c426b7007442e8240c117f
Lung_Opacity-1733.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1733.png,Lung_Opacity/images/Lung_Opacity-1733.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1733.png,Lung_Opacity/masks/Lung_Opacity-1733.png,True,False,299,299,L,125.67,1a13d52c65c1bc371930f89af904f8f6
Lung_Opacity-1734.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1734.png,Lung_Opacity/images/Lung_Opacity-1734.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1734.png,Lung_Opacity/masks/Lung_Opacity-1734.png,True,False,299,299,L,102.42,b24a085fdedffe5986e963322af2abd5
Lung_Opacity-1735.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1735.png,Lung_Opacity/images/Lung_Opacity-1735.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1735.png,Lung_Opacity/masks/Lung_Opacity-1735.png,True,False,299,299,L,138.49,0ef4c304635e9fce9e8a0c2f798bbfb1
Lung_Opacity-1736.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1736.png,Lung_Opacity/images/Lung_Opacity-1736.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1736.png,Lung_Opacity/masks/Lung_Opacity-1736.png,True,False,299,299,L,148.42,70466d6ec2c5f4d865cd20071799d984
Lung_Opacity-1737.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1737.png,Lung_Opacity/images/Lung_Opacity-1737.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1737.png,Lung_Opacity/masks/Lung_Opacity-1737.png,True,False,299,299,L,117.97,3738637d04799503b5aec1ad6750fc91
Lung_Opacity-1738.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1738.png,Lung_Opacity/images/Lung_Opacity-1738.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1738.png,Lung_Opacity/masks/Lung_Opacity-1738.png,True,False,299,299,L,120.24,fc15948332021f77490584df1fb1376f
Lung_Opacity-1739.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1739.png,Lung_Opacity/images/Lung_Opacity-1739.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1739.png,Lung_Opacity/masks/Lung_Opacity-1739.png,True,False,299,299,L,131.44,14128e0f111518c67c3d5a8d10b11066
Lung_Opacity-174.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-174.png,Lung_Opacity/images/Lung_Opacity-174.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-174.png,Lung_Opacity/masks/Lung_Opacity-174.png,True,False,299,299,L,117.09,e1d4b7c466486b99bc855c39e6c49177
Lung_Opacity-1740.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1740.png,Lung_Opacity/images/Lung_Opacity-1740.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1740.png,Lung_Opacity/masks/Lung_Opacity-1740.png,True,False,299,299,L,113.96,21153b314a2063c70125b0639209b2d4
Lung_Opacity-1741.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1741.png,Lung_Opacity/images/Lung_Opacity-1741.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1741.png,Lung_Opacity/masks/Lung_Opacity-1741.png,True,False,299,299,L,159.87,0df184beb540a8ffd28509d587f77337
Lung_Opacity-1742.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1742.png,Lung_Opacity/images/Lung_Opacity-1742.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1742.png,Lung_Opacity/masks/Lung_Opacity-1742.png,True,False,299,299,L,107.02,55e08ec200c8766036c2511bd28b2249
Lung_Opacity-1743.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1743.png,Lung_Opacity/images/Lung_Opacity-1743.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1743.png,Lung_Opacity/masks/Lung_Opacity-1743.png,True,False,299,299,L,121.32,242cb3665d14d724bffe2832f4643bf9
Lung_Opacity-1744.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1744.png,Lung_Opacity/images/Lung_Opacity-1744.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1744.png,Lung_Opacity/masks/Lung_Opacity-1744.png,True,False,299,299,L,122.7,3cc2e43a0307ca882f7c61b3190b8a4e
Lung_Opacity-1745.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1745.png,Lung_Opacity/images/Lung_Opacity-1745.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1745.png,Lung_Opacity/masks/Lung_Opacity-1745.png,True,False,299,299,L,101.76,8887889de60af6dcfe438e7d971dcd65
Lung_Opacity-1746.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1746.png,Lung_Opacity/images/Lung_Opacity-1746.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1746.png,Lung_Opacity/masks/Lung_Opacity-1746.png,True,False,299,299,L,93.46,92308ffcd07f907031dab46c12f50419
Lung_Opacity-1747.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1747.png,Lung_Opacity/images/Lung_Opacity-1747.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1747.png,Lung_Opacity/masks/Lung_Opacity-1747.png,True,False,299,299,L,178.64,054e2868371bbe0973c36409ab8173f7
Lung_Opacity-1748.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1748.png,Lung_Opacity/images/Lung_Opacity-1748.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1748.png,Lung_Opacity/masks/Lung_Opacity-1748.png,True,False,299,299,L,141.85,5733460a7699922fc4ae48d9e6a6f8ea
Lung_Opacity-1749.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1749.png,Lung_Opacity/images/Lung_Opacity-1749.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1749.png,Lung_Opacity/masks/Lung_Opacity-1749.png,True,False,299,299,L,115.04,4b26611d4c9532541c3db5d63a33fc1b
Lung_Opacity-175.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-175.png,Lung_Opacity/images/Lung_Opacity-175.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-175.png,Lung_Opacity/masks/Lung_Opacity-175.png,True,False,299,299,L,123.49,f148d46f588afe59ff89bb142610e28e
Lung_Opacity-1750.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1750.png,Lung_Opacity/images/Lung_Opacity-1750.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1750.png,Lung_Opacity/masks/Lung_Opacity-1750.png,True,False,299,299,L,148.52,fdb0ad574a35825ef87af10ecb4f951b
Lung_Opacity-1751.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1751.png,Lung_Opacity/images/Lung_Opacity-1751.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1751.png,Lung_Opacity/masks/Lung_Opacity-1751.png,True,False,299,299,L,125.65,c5a52dc9ca1087e21e610ba719f9f165
Lung_Opacity-1752.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1752.png,Lung_Opacity/images/Lung_Opacity-1752.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1752.png,Lung_Opacity/masks/Lung_Opacity-1752.png,True,False,299,299,L,172.02,02bd4f4e8c76845f407315f7fca8def9
Lung_Opacity-1753.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1753.png,Lung_Opacity/images/Lung_Opacity-1753.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1753.png,Lung_Opacity/masks/Lung_Opacity-1753.png,True,False,299,299,L,121.59,4b361189b7eacaf4b9edcbba1915c457
Lung_Opacity-1754.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1754.png,Lung_Opacity/images/Lung_Opacity-1754.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1754.png,Lung_Opacity/masks/Lung_Opacity-1754.png,True,False,299,299,L,145.55,04f99af94a7341e4f099eed01704b4b7
Lung_Opacity-1755.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1755.png,Lung_Opacity/images/Lung_Opacity-1755.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1755.png,Lung_Opacity/masks/Lung_Opacity-1755.png,True,False,299,299,L,132.72,9338cb0681f6dbf2550be625ba1c8d8b
Lung_Opacity-1756.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1756.png,Lung_Opacity/images/Lung_Opacity-1756.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1756.png,Lung_Opacity/masks/Lung_Opacity-1756.png,True,False,299,299,L,71.75,e2be86d50e883fe1975861a3f8265878
Lung_Opacity-1757.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1757.png,Lung_Opacity/images/Lung_Opacity-1757.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1757.png,Lung_Opacity/masks/Lung_Opacity-1757.png,True,False,299,299,L,131.85,e6f4e69045ed676ce41fb16f308ccd1d
Lung_Opacity-1758.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1758.png,Lung_Opacity/images/Lung_Opacity-1758.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1758.png,Lung_Opacity/masks/Lung_Opacity-1758.png,True,False,299,299,L,108.12,7349a989aff4388182e40b0f11c676f1
Lung_Opacity-1759.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1759.png,Lung_Opacity/images/Lung_Opacity-1759.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1759.png,Lung_Opacity/masks/Lung_Opacity-1759.png,True,False,299,299,L,159.5,a0c07724597e132f8148089c4d97e90b
Lung_Opacity-176.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-176.png,Lung_Opacity/images/Lung_Opacity-176.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-176.png,Lung_Opacity/masks/Lung_Opacity-176.png,True,False,299,299,L,130.79,7b9130e58d22e3286db898fc27fcbcc1
Lung_Opacity-1760.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1760.png,Lung_Opacity/images/Lung_Opacity-1760.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1760.png,Lung_Opacity/masks/Lung_Opacity-1760.png,True,False,299,299,L,117.12,a98f69b68165af6301fd24f4706c821a
Lung_Opacity-1761.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1761.png,Lung_Opacity/images/Lung_Opacity-1761.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1761.png,Lung_Opacity/masks/Lung_Opacity-1761.png,True,False,299,299,L,114.51,b73c2865c650d9fa07beb05b3abca6e6
Lung_Opacity-1762.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1762.png,Lung_Opacity/images/Lung_Opacity-1762.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1762.png,Lung_Opacity/masks/Lung_Opacity-1762.png,True,False,299,299,L,111.11,85c8d2be33f77883acc8e3129c1080c9
Lung_Opacity-1763.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1763.png,Lung_Opacity/images/Lung_Opacity-1763.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1763.png,Lung_Opacity/masks/Lung_Opacity-1763.png,True,False,299,299,L,193.66,caf4981883bbda76be39ba0a538cc866
Lung_Opacity-1764.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1764.png,Lung_Opacity/images/Lung_Opacity-1764.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1764.png,Lung_Opacity/masks/Lung_Opacity-1764.png,True,False,299,299,L,113.24,c48518f61ab5fc902c55d9a06c716f3b
Lung_Opacity-1765.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1765.png,Lung_Opacity/images/Lung_Opacity-1765.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1765.png,Lung_Opacity/masks/Lung_Opacity-1765.png,True,False,299,299,L,178.44,62fdd197abd8c38b00910348d5a992ab
Lung_Opacity-1766.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1766.png,Lung_Opacity/images/Lung_Opacity-1766.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1766.png,Lung_Opacity/masks/Lung_Opacity-1766.png,True,False,299,299,L,115.27,61e8f5092dfd33e6a623f8ce4358a662
Lung_Opacity-1767.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1767.png,Lung_Opacity/images/Lung_Opacity-1767.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1767.png,Lung_Opacity/masks/Lung_Opacity-1767.png,True,False,299,299,L,156.96,850eae4f0789747159c1ac62388a3a87
Lung_Opacity-1768.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1768.png,Lung_Opacity/images/Lung_Opacity-1768.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1768.png,Lung_Opacity/masks/Lung_Opacity-1768.png,True,False,299,299,L,120.67,a5e631b84dc874cd6ef19ee5030dd55e
Lung_Opacity-1769.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1769.png,Lung_Opacity/images/Lung_Opacity-1769.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1769.png,Lung_Opacity/masks/Lung_Opacity-1769.png,True,False,299,299,L,127.1,aa1e76effea00cae6663875659d57465
Lung_Opacity-177.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-177.png,Lung_Opacity/images/Lung_Opacity-177.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-177.png,Lung_Opacity/masks/Lung_Opacity-177.png,True,False,299,299,L,124.98,3c7d10842621942ee315b6e2f56b4b9b
Lung_Opacity-1770.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1770.png,Lung_Opacity/images/Lung_Opacity-1770.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1770.png,Lung_Opacity/masks/Lung_Opacity-1770.png,True,False,299,299,L,105.1,ae984706a11fb88df72836684672171a
Lung_Opacity-1771.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1771.png,Lung_Opacity/images/Lung_Opacity-1771.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1771.png,Lung_Opacity/masks/Lung_Opacity-1771.png,True,False,299,299,L,125.73,4093f73509105b0d569781f0bcb27e3e
Lung_Opacity-1772.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1772.png,Lung_Opacity/images/Lung_Opacity-1772.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1772.png,Lung_Opacity/masks/Lung_Opacity-1772.png,True,False,299,299,L,117.5,a70ca647d2e6e1e521c1a3d3496e280b
Lung_Opacity-1773.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1773.png,Lung_Opacity/images/Lung_Opacity-1773.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1773.png,Lung_Opacity/masks/Lung_Opacity-1773.png,True,False,299,299,L,157.57,0cf6f559e853ebff6689a9931ddba421
Lung_Opacity-1774.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1774.png,Lung_Opacity/images/Lung_Opacity-1774.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1774.png,Lung_Opacity/masks/Lung_Opacity-1774.png,True,False,299,299,L,98.12,7612f436b59cc992c787be7dba7855a2
Lung_Opacity-1775.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1775.png,Lung_Opacity/images/Lung_Opacity-1775.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1775.png,Lung_Opacity/masks/Lung_Opacity-1775.png,True,False,299,299,L,115.32,f0a233bc44c3f5240fe945b10988fd66
Lung_Opacity-1776.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1776.png,Lung_Opacity/images/Lung_Opacity-1776.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1776.png,Lung_Opacity/masks/Lung_Opacity-1776.png,True,False,299,299,L,57.54,11cefdec77fcbb1677f6d8dfdcf89d9f
Lung_Opacity-1777.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1777.png,Lung_Opacity/images/Lung_Opacity-1777.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1777.png,Lung_Opacity/masks/Lung_Opacity-1777.png,True,False,299,299,L,115.4,5f5991204e9317eb2e4f00e27706cf27
Lung_Opacity-1778.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1778.png,Lung_Opacity/images/Lung_Opacity-1778.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1778.png,Lung_Opacity/masks/Lung_Opacity-1778.png,True,False,299,299,L,107.44,79b0f3fe9defec384c8296d670c2a1a9
Lung_Opacity-1779.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1779.png,Lung_Opacity/images/Lung_Opacity-1779.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1779.png,Lung_Opacity/masks/Lung_Opacity-1779.png,True,False,299,299,L,103.4,e94efbf23cb6cc366dfdb15339feaa7c
Lung_Opacity-178.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-178.png,Lung_Opacity/images/Lung_Opacity-178.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-178.png,Lung_Opacity/masks/Lung_Opacity-178.png,True,False,299,299,L,116.34,c9fd0fe0bc8268773bec91c3012a5223
Lung_Opacity-1780.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1780.png,Lung_Opacity/images/Lung_Opacity-1780.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1780.png,Lung_Opacity/masks/Lung_Opacity-1780.png,True,False,299,299,L,133.88,7ded0b82f18ef9145e423d24cfa63b12
Lung_Opacity-1781.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1781.png,Lung_Opacity/images/Lung_Opacity-1781.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1781.png,Lung_Opacity/masks/Lung_Opacity-1781.png,True,False,299,299,L,105.38,c5555a9d3a2b3568d97a3d87e6b897db
Lung_Opacity-1782.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1782.png,Lung_Opacity/images/Lung_Opacity-1782.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1782.png,Lung_Opacity/masks/Lung_Opacity-1782.png,True,False,299,299,L,142.55,7c47e22c4c31e7fd506d8da1e9728f79
Lung_Opacity-1783.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1783.png,Lung_Opacity/images/Lung_Opacity-1783.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1783.png,Lung_Opacity/masks/Lung_Opacity-1783.png,True,False,299,299,L,131.59,22728c788a79322db2ae61f2cb533572
Lung_Opacity-1784.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1784.png,Lung_Opacity/images/Lung_Opacity-1784.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1784.png,Lung_Opacity/masks/Lung_Opacity-1784.png,True,False,299,299,L,146.12,f540eded6bab8e91f86b18908999edb5
Lung_Opacity-1785.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1785.png,Lung_Opacity/images/Lung_Opacity-1785.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1785.png,Lung_Opacity/masks/Lung_Opacity-1785.png,True,False,299,299,L,158.62,6ee54e788e47763af3a851092628f849
Lung_Opacity-1786.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1786.png,Lung_Opacity/images/Lung_Opacity-1786.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1786.png,Lung_Opacity/masks/Lung_Opacity-1786.png,True,False,299,299,L,180.06,c44b1778b062b2bff4817dafa66c34c0
Lung_Opacity-1787.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1787.png,Lung_Opacity/images/Lung_Opacity-1787.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1787.png,Lung_Opacity/masks/Lung_Opacity-1787.png,True,False,299,299,L,164.21,374fd8f3402aa404d5d67a5dcc536aa1
Lung_Opacity-1788.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1788.png,Lung_Opacity/images/Lung_Opacity-1788.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1788.png,Lung_Opacity/masks/Lung_Opacity-1788.png,True,False,299,299,L,121.53,e0e56daeabe8fa46506390ab419bd178
Lung_Opacity-1789.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1789.png,Lung_Opacity/images/Lung_Opacity-1789.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1789.png,Lung_Opacity/masks/Lung_Opacity-1789.png,True,False,299,299,L,111.74,9fc0e36db890a6f528b54a97ca89569b
Lung_Opacity-179.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-179.png,Lung_Opacity/images/Lung_Opacity-179.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-179.png,Lung_Opacity/masks/Lung_Opacity-179.png,True,False,299,299,L,120.16,02b7e123fe5561f3e23e97334f7b8afe
Lung_Opacity-1790.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1790.png,Lung_Opacity/images/Lung_Opacity-1790.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1790.png,Lung_Opacity/masks/Lung_Opacity-1790.png,True,False,299,299,L,117.11,92248dda1fe9ab18f6f5c58239ad0156
Lung_Opacity-1791.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1791.png,Lung_Opacity/images/Lung_Opacity-1791.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1791.png,Lung_Opacity/masks/Lung_Opacity-1791.png,True,False,299,299,L,136.58,6fbb47e5d8bc4c419d0f104bda6743c5
Lung_Opacity-1792.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1792.png,Lung_Opacity/images/Lung_Opacity-1792.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1792.png,Lung_Opacity/masks/Lung_Opacity-1792.png,True,False,299,299,L,108.91,649c79936b03373c0c36f189bfcfb3f9
Lung_Opacity-1793.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1793.png,Lung_Opacity/images/Lung_Opacity-1793.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1793.png,Lung_Opacity/masks/Lung_Opacity-1793.png,True,False,299,299,L,90.87,ce93acb1a283854e7e1f0c25a47b111c
Lung_Opacity-1794.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1794.png,Lung_Opacity/images/Lung_Opacity-1794.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1794.png,Lung_Opacity/masks/Lung_Opacity-1794.png,True,False,299,299,L,114.71,23c20b1e2e460b58cc4e51b4df14cb87
Lung_Opacity-1795.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1795.png,Lung_Opacity/images/Lung_Opacity-1795.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1795.png,Lung_Opacity/masks/Lung_Opacity-1795.png,True,False,299,299,L,114.78,58ae59d5ed98b96d682db25a8704d33f
Lung_Opacity-1796.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1796.png,Lung_Opacity/images/Lung_Opacity-1796.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1796.png,Lung_Opacity/masks/Lung_Opacity-1796.png,True,False,299,299,L,145.08,7a03a036eee1791f92d9ded833131ef3
Lung_Opacity-1797.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1797.png,Lung_Opacity/images/Lung_Opacity-1797.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1797.png,Lung_Opacity/masks/Lung_Opacity-1797.png,True,False,299,299,L,106.72,6116f49ae02ecc87f0261d9a5890ff55
Lung_Opacity-1798.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1798.png,Lung_Opacity/images/Lung_Opacity-1798.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1798.png,Lung_Opacity/masks/Lung_Opacity-1798.png,True,False,299,299,L,116.14,4031e9dc5cb12a0979451d8e3b717939
Lung_Opacity-1799.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1799.png,Lung_Opacity/images/Lung_Opacity-1799.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1799.png,Lung_Opacity/masks/Lung_Opacity-1799.png,True,False,299,299,L,98.92,b34ce872466bb46be7dbd0f4f8e30fe2
Lung_Opacity-18.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-18.png,Lung_Opacity/images/Lung_Opacity-18.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-18.png,Lung_Opacity/masks/Lung_Opacity-18.png,True,False,299,299,L,114.16,c13bfb924f8f9f2131b3eba930d065b1
Lung_Opacity-180.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-180.png,Lung_Opacity/images/Lung_Opacity-180.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-180.png,Lung_Opacity/masks/Lung_Opacity-180.png,True,False,299,299,L,165.9,e619945bc83e1d6ea4a4535f6a02b61b
Lung_Opacity-1800.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1800.png,Lung_Opacity/images/Lung_Opacity-1800.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1800.png,Lung_Opacity/masks/Lung_Opacity-1800.png,True,False,299,299,L,133.32,0dce453d740c6871d7daa726b53d3013
Lung_Opacity-1801.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1801.png,Lung_Opacity/images/Lung_Opacity-1801.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1801.png,Lung_Opacity/masks/Lung_Opacity-1801.png,True,False,299,299,L,133.97,2ae32b1387c0c4580cb42b83453af0fe
Lung_Opacity-1802.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1802.png,Lung_Opacity/images/Lung_Opacity-1802.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1802.png,Lung_Opacity/masks/Lung_Opacity-1802.png,True,False,299,299,L,114.66,a5d3f2503c10bd6629da9301459f2fe0
Lung_Opacity-1803.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1803.png,Lung_Opacity/images/Lung_Opacity-1803.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1803.png,Lung_Opacity/masks/Lung_Opacity-1803.png,True,False,299,299,L,148.98,dc3b41d19cdc05fe5b54531efe856841
Lung_Opacity-1804.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1804.png,Lung_Opacity/images/Lung_Opacity-1804.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1804.png,Lung_Opacity/masks/Lung_Opacity-1804.png,True,False,299,299,L,112.22,2523ded0f481c46cb20a767ad7ccd067
Lung_Opacity-1805.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1805.png,Lung_Opacity/images/Lung_Opacity-1805.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1805.png,Lung_Opacity/masks/Lung_Opacity-1805.png,True,False,299,299,L,116.43,7964b2074ff9a433b09a524c014db6ee
Lung_Opacity-1806.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1806.png,Lung_Opacity/images/Lung_Opacity-1806.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1806.png,Lung_Opacity/masks/Lung_Opacity-1806.png,True,False,299,299,L,100.07,17512e36d650d2992ccc8806f60452af
Lung_Opacity-1807.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1807.png,Lung_Opacity/images/Lung_Opacity-1807.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1807.png,Lung_Opacity/masks/Lung_Opacity-1807.png,True,False,299,299,L,162.97,c83b7015473f8afe8a65ac262175ba89
Lung_Opacity-1808.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1808.png,Lung_Opacity/images/Lung_Opacity-1808.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1808.png,Lung_Opacity/masks/Lung_Opacity-1808.png,True,False,299,299,L,92.97,4a17b47a035fa4ed4ec0f406a1cc3d5f
Lung_Opacity-1809.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1809.png,Lung_Opacity/images/Lung_Opacity-1809.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1809.png,Lung_Opacity/masks/Lung_Opacity-1809.png,True,False,299,299,L,169.37,d4960326b66f025396d1120334d78dec
Lung_Opacity-181.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-181.png,Lung_Opacity/images/Lung_Opacity-181.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-181.png,Lung_Opacity/masks/Lung_Opacity-181.png,True,False,299,299,L,143.98,69b40b65ee71faa6b832cda045ce9ad9
Lung_Opacity-1810.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1810.png,Lung_Opacity/images/Lung_Opacity-1810.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1810.png,Lung_Opacity/masks/Lung_Opacity-1810.png,True,False,299,299,L,149.45,dfd3cc7c29843ac5e9ece29f561e30ba
Lung_Opacity-1811.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1811.png,Lung_Opacity/images/Lung_Opacity-1811.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1811.png,Lung_Opacity/masks/Lung_Opacity-1811.png,True,False,299,299,L,121.18,ba425054865ebc1e724d72d7e251c088
Lung_Opacity-1812.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1812.png,Lung_Opacity/images/Lung_Opacity-1812.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1812.png,Lung_Opacity/masks/Lung_Opacity-1812.png,True,False,299,299,L,100.5,76899a4f81203c0cc4039101b9015d65
Lung_Opacity-1813.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1813.png,Lung_Opacity/images/Lung_Opacity-1813.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1813.png,Lung_Opacity/masks/Lung_Opacity-1813.png,True,False,299,299,L,122.46,11fba6283b0812d83dbb9e4571038eb9
Lung_Opacity-1814.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1814.png,Lung_Opacity/images/Lung_Opacity-1814.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1814.png,Lung_Opacity/masks/Lung_Opacity-1814.png,True,False,299,299,L,140.13,f42f9726c0d277db4dc84f68932a7cc9
Lung_Opacity-1815.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1815.png,Lung_Opacity/images/Lung_Opacity-1815.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1815.png,Lung_Opacity/masks/Lung_Opacity-1815.png,True,False,299,299,L,168.24,70a1f108dc3fbfd31c2dfecde9c859a1
Lung_Opacity-1816.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1816.png,Lung_Opacity/images/Lung_Opacity-1816.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1816.png,Lung_Opacity/masks/Lung_Opacity-1816.png,True,False,299,299,L,133.88,7cbb67ad4a39cf20081431345ecd71fe
Lung_Opacity-1817.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1817.png,Lung_Opacity/images/Lung_Opacity-1817.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1817.png,Lung_Opacity/masks/Lung_Opacity-1817.png,True,False,299,299,L,146.64,18695222f2c1e9f581e84471edc0988a
Lung_Opacity-1818.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1818.png,Lung_Opacity/images/Lung_Opacity-1818.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1818.png,Lung_Opacity/masks/Lung_Opacity-1818.png,True,False,299,299,L,175.16,90e29c08cef380f2c07f455a45f85e04
Lung_Opacity-1819.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1819.png,Lung_Opacity/images/Lung_Opacity-1819.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1819.png,Lung_Opacity/masks/Lung_Opacity-1819.png,True,False,299,299,L,155.31,514d5104ac501080d341be63f609b870
Lung_Opacity-182.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-182.png,Lung_Opacity/images/Lung_Opacity-182.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-182.png,Lung_Opacity/masks/Lung_Opacity-182.png,True,False,299,299,L,150.53,5c2c8915946102ae1da70ac6e144665c
Lung_Opacity-1820.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1820.png,Lung_Opacity/images/Lung_Opacity-1820.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1820.png,Lung_Opacity/masks/Lung_Opacity-1820.png,True,False,299,299,L,140.78,d86d5eb36410360a0c16191a8d3fac38
Lung_Opacity-1821.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1821.png,Lung_Opacity/images/Lung_Opacity-1821.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1821.png,Lung_Opacity/masks/Lung_Opacity-1821.png,True,False,299,299,L,162.27,c92cb130cad20101e760ce480298f82e
Lung_Opacity-1822.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1822.png,Lung_Opacity/images/Lung_Opacity-1822.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1822.png,Lung_Opacity/masks/Lung_Opacity-1822.png,True,False,299,299,L,133.97,b1262c02f1385b604620331be96ef9dc
Lung_Opacity-1823.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1823.png,Lung_Opacity/images/Lung_Opacity-1823.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1823.png,Lung_Opacity/masks/Lung_Opacity-1823.png,True,False,299,299,L,148.53,40cec64bde97851d2ee981a50c225505
Lung_Opacity-1824.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1824.png,Lung_Opacity/images/Lung_Opacity-1824.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1824.png,Lung_Opacity/masks/Lung_Opacity-1824.png,True,False,299,299,L,120.05,ac7b328b7d361a3b60bff32834d09fca
Lung_Opacity-1825.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1825.png,Lung_Opacity/images/Lung_Opacity-1825.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1825.png,Lung_Opacity/masks/Lung_Opacity-1825.png,True,False,299,299,L,152.13,3c730892c131e83982d509c8966336bc
Lung_Opacity-1826.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1826.png,Lung_Opacity/images/Lung_Opacity-1826.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1826.png,Lung_Opacity/masks/Lung_Opacity-1826.png,True,False,299,299,L,161.56,c175711f176d41ec5f9784cb1d5a6fcf
Lung_Opacity-1827.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1827.png,Lung_Opacity/images/Lung_Opacity-1827.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1827.png,Lung_Opacity/masks/Lung_Opacity-1827.png,True,False,299,299,L,137.55,e5b57e2bfa387b1b4fd9a2e160ba6726
Lung_Opacity-1828.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1828.png,Lung_Opacity/images/Lung_Opacity-1828.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1828.png,Lung_Opacity/masks/Lung_Opacity-1828.png,True,False,299,299,L,101.82,47eec52659cb21e0ab87870908fa2232
Lung_Opacity-1829.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1829.png,Lung_Opacity/images/Lung_Opacity-1829.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1829.png,Lung_Opacity/masks/Lung_Opacity-1829.png,True,False,299,299,L,123.24,c9ad80e5bc3d008c85ef4cbaef32b6e3
Lung_Opacity-183.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-183.png,Lung_Opacity/images/Lung_Opacity-183.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-183.png,Lung_Opacity/masks/Lung_Opacity-183.png,True,False,299,299,L,139.35,329f4da38dacf74023220e752497cfd6
Lung_Opacity-1830.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1830.png,Lung_Opacity/images/Lung_Opacity-1830.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1830.png,Lung_Opacity/masks/Lung_Opacity-1830.png,True,False,299,299,L,122.12,a277b5aae1159ff9f5617191974d9146
Lung_Opacity-1831.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1831.png,Lung_Opacity/images/Lung_Opacity-1831.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1831.png,Lung_Opacity/masks/Lung_Opacity-1831.png,True,False,299,299,L,118.34,879f689a2590474fe646b806482038a4
Lung_Opacity-1832.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1832.png,Lung_Opacity/images/Lung_Opacity-1832.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1832.png,Lung_Opacity/masks/Lung_Opacity-1832.png,True,False,299,299,L,108.92,19656e50eea0ccf92ddfd7025a96d871
Lung_Opacity-1833.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1833.png,Lung_Opacity/images/Lung_Opacity-1833.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1833.png,Lung_Opacity/masks/Lung_Opacity-1833.png,True,False,299,299,L,130.97,7b472bbba175cec056e4da12b4daae38
Lung_Opacity-1834.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1834.png,Lung_Opacity/images/Lung_Opacity-1834.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1834.png,Lung_Opacity/masks/Lung_Opacity-1834.png,True,False,299,299,L,109.73,2b045646dbc7c16824cee0edb09ffe2b
Lung_Opacity-1835.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1835.png,Lung_Opacity/images/Lung_Opacity-1835.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1835.png,Lung_Opacity/masks/Lung_Opacity-1835.png,True,False,299,299,L,181.12,955466885b885c041da03185a8314bb5
Lung_Opacity-1836.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1836.png,Lung_Opacity/images/Lung_Opacity-1836.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1836.png,Lung_Opacity/masks/Lung_Opacity-1836.png,True,False,299,299,L,115.78,63a5658d1ee8e865bd37e5ac60a5591c
Lung_Opacity-1837.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1837.png,Lung_Opacity/images/Lung_Opacity-1837.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1837.png,Lung_Opacity/masks/Lung_Opacity-1837.png,True,False,299,299,L,118.68,ed67c719a6d7192cbbd1e3e77e2aa9ae
Lung_Opacity-1838.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1838.png,Lung_Opacity/images/Lung_Opacity-1838.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1838.png,Lung_Opacity/masks/Lung_Opacity-1838.png,True,False,299,299,L,130.48,8b02751d3a036d2cc46b018a54e0838a
Lung_Opacity-1839.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1839.png,Lung_Opacity/images/Lung_Opacity-1839.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1839.png,Lung_Opacity/masks/Lung_Opacity-1839.png,True,False,299,299,L,157.56,4ec15d57c5786eab516141d04bc1f638
Lung_Opacity-184.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-184.png,Lung_Opacity/images/Lung_Opacity-184.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-184.png,Lung_Opacity/masks/Lung_Opacity-184.png,True,False,299,299,L,118.32,6749b9d2eb5302ea4c6457d9b374a86e
Lung_Opacity-1840.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1840.png,Lung_Opacity/images/Lung_Opacity-1840.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1840.png,Lung_Opacity/masks/Lung_Opacity-1840.png,True,False,299,299,L,149.46,675ebed9c859ffa3e75117b7c4d1179c
Lung_Opacity-1841.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1841.png,Lung_Opacity/images/Lung_Opacity-1841.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1841.png,Lung_Opacity/masks/Lung_Opacity-1841.png,True,False,299,299,L,112.51,0434cf7d5fc8b358097a82aaaf866b12
Lung_Opacity-1842.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1842.png,Lung_Opacity/images/Lung_Opacity-1842.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1842.png,Lung_Opacity/masks/Lung_Opacity-1842.png,True,False,299,299,L,116.52,bd1fbf4310ae5d5ed37001352d23c964
Lung_Opacity-1843.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1843.png,Lung_Opacity/images/Lung_Opacity-1843.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1843.png,Lung_Opacity/masks/Lung_Opacity-1843.png,True,False,299,299,L,163.13,c0811fb7d02b7b29d5095a1c5aac7407
Lung_Opacity-1844.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1844.png,Lung_Opacity/images/Lung_Opacity-1844.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1844.png,Lung_Opacity/masks/Lung_Opacity-1844.png,True,False,299,299,L,100.0,e46c68e09ee2f4d540fbc64b32a02069
Lung_Opacity-1845.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1845.png,Lung_Opacity/images/Lung_Opacity-1845.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1845.png,Lung_Opacity/masks/Lung_Opacity-1845.png,True,False,299,299,L,103.57,ca58e5d231cebd2ada0b4a4ff1e600e3
Lung_Opacity-1846.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1846.png,Lung_Opacity/images/Lung_Opacity-1846.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1846.png,Lung_Opacity/masks/Lung_Opacity-1846.png,True,False,299,299,L,134.12,e01c18168a0951469589af95a4f9b339
Lung_Opacity-1847.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1847.png,Lung_Opacity/images/Lung_Opacity-1847.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1847.png,Lung_Opacity/masks/Lung_Opacity-1847.png,True,False,299,299,L,94.51,ff7eb7a228f02e140e508120bcfc770f
Lung_Opacity-1848.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1848.png,Lung_Opacity/images/Lung_Opacity-1848.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1848.png,Lung_Opacity/masks/Lung_Opacity-1848.png,True,False,299,299,L,103.18,11231d973801d7cf19ba430f75b7c9cc
Lung_Opacity-1849.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1849.png,Lung_Opacity/images/Lung_Opacity-1849.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1849.png,Lung_Opacity/masks/Lung_Opacity-1849.png,True,False,299,299,L,119.84,9d1ffdb09dec22fb3fbd041ba96a046a
Lung_Opacity-185.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-185.png,Lung_Opacity/images/Lung_Opacity-185.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-185.png,Lung_Opacity/masks/Lung_Opacity-185.png,True,False,299,299,L,159.83,e9258baf53e054108afb58a4569d62f1
Lung_Opacity-1850.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1850.png,Lung_Opacity/images/Lung_Opacity-1850.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1850.png,Lung_Opacity/masks/Lung_Opacity-1850.png,True,False,299,299,L,132.9,cb01ee9917b3403562195e9d012e9990
Lung_Opacity-1851.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1851.png,Lung_Opacity/images/Lung_Opacity-1851.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1851.png,Lung_Opacity/masks/Lung_Opacity-1851.png,True,False,299,299,L,106.84,7495626562af0d831fc7130c21f67f5f
Lung_Opacity-1852.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1852.png,Lung_Opacity/images/Lung_Opacity-1852.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1852.png,Lung_Opacity/masks/Lung_Opacity-1852.png,True,False,299,299,L,161.9,995c0d013cc957b6a44fd3d1ba71124e
Lung_Opacity-1853.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1853.png,Lung_Opacity/images/Lung_Opacity-1853.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1853.png,Lung_Opacity/masks/Lung_Opacity-1853.png,True,False,299,299,L,160.36,22d4c8991047a4b3ed03042e109e4381
Lung_Opacity-1854.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1854.png,Lung_Opacity/images/Lung_Opacity-1854.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1854.png,Lung_Opacity/masks/Lung_Opacity-1854.png,True,False,299,299,L,111.21,1470a80bb46376b436c88ff4076857ea
Lung_Opacity-1855.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1855.png,Lung_Opacity/images/Lung_Opacity-1855.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1855.png,Lung_Opacity/masks/Lung_Opacity-1855.png,True,False,299,299,L,98.07,411fa0b7fba93c02482c4235032b3570
Lung_Opacity-1856.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1856.png,Lung_Opacity/images/Lung_Opacity-1856.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1856.png,Lung_Opacity/masks/Lung_Opacity-1856.png,True,False,299,299,L,122.13,f28bca8778fef748d30cbde683ed2591
Lung_Opacity-1857.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1857.png,Lung_Opacity/images/Lung_Opacity-1857.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1857.png,Lung_Opacity/masks/Lung_Opacity-1857.png,True,False,299,299,L,124.91,224d44644bd364b920e1defb4392e617
Lung_Opacity-1858.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1858.png,Lung_Opacity/images/Lung_Opacity-1858.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1858.png,Lung_Opacity/masks/Lung_Opacity-1858.png,True,False,299,299,L,111.88,2b64d359b1dc8465ce02d58bff164925
Lung_Opacity-1859.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1859.png,Lung_Opacity/images/Lung_Opacity-1859.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1859.png,Lung_Opacity/masks/Lung_Opacity-1859.png,True,False,299,299,L,109.63,a7254ce76e480319eea3123272ce346c
Lung_Opacity-186.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-186.png,Lung_Opacity/images/Lung_Opacity-186.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-186.png,Lung_Opacity/masks/Lung_Opacity-186.png,True,False,299,299,L,99.42,10450ef8cf5f14c4134729d302fac6bc
Lung_Opacity-1860.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1860.png,Lung_Opacity/images/Lung_Opacity-1860.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1860.png,Lung_Opacity/masks/Lung_Opacity-1860.png,True,False,299,299,L,135.13,6abc432f2be93662dec8a0b0e0d1e66e
Lung_Opacity-1861.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1861.png,Lung_Opacity/images/Lung_Opacity-1861.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1861.png,Lung_Opacity/masks/Lung_Opacity-1861.png,True,False,299,299,L,119.46,e6f2e1224dce0300b003b3063c880bbb
Lung_Opacity-1862.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1862.png,Lung_Opacity/images/Lung_Opacity-1862.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1862.png,Lung_Opacity/masks/Lung_Opacity-1862.png,True,False,299,299,L,91.94,d5fea4e63a7c8168c6c6c98cd5dfd7ee
Lung_Opacity-1863.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1863.png,Lung_Opacity/images/Lung_Opacity-1863.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1863.png,Lung_Opacity/masks/Lung_Opacity-1863.png,True,False,299,299,L,139.06,3befb54112cdc63707d242d6b61b5dec
Lung_Opacity-1864.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1864.png,Lung_Opacity/images/Lung_Opacity-1864.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1864.png,Lung_Opacity/masks/Lung_Opacity-1864.png,True,False,299,299,L,108.52,0c4ab85a2d95d670f92e45d30c5df6cb
Lung_Opacity-1865.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1865.png,Lung_Opacity/images/Lung_Opacity-1865.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1865.png,Lung_Opacity/masks/Lung_Opacity-1865.png,True,False,299,299,L,123.1,bccf288286c6eb1f4fb2672bb1b62a60
Lung_Opacity-1866.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1866.png,Lung_Opacity/images/Lung_Opacity-1866.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1866.png,Lung_Opacity/masks/Lung_Opacity-1866.png,True,False,299,299,L,158.5,82ada0385271619e9987ce2a7b666499
Lung_Opacity-1867.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1867.png,Lung_Opacity/images/Lung_Opacity-1867.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1867.png,Lung_Opacity/masks/Lung_Opacity-1867.png,True,False,299,299,L,115.23,8df75f9601885c7b1cac9013b43778e3
Lung_Opacity-1868.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1868.png,Lung_Opacity/images/Lung_Opacity-1868.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1868.png,Lung_Opacity/masks/Lung_Opacity-1868.png,True,False,299,299,L,119.44,ddc1d9f9e16f873cbdb93847086ad85f
Lung_Opacity-1869.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1869.png,Lung_Opacity/images/Lung_Opacity-1869.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1869.png,Lung_Opacity/masks/Lung_Opacity-1869.png,True,False,299,299,L,102.46,3b9d9cda6342f786fe2e6b3947923cc3
Lung_Opacity-187.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-187.png,Lung_Opacity/images/Lung_Opacity-187.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-187.png,Lung_Opacity/masks/Lung_Opacity-187.png,True,False,299,299,L,133.81,34d7475a8d4f4d545c015558806e0bde
Lung_Opacity-1870.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1870.png,Lung_Opacity/images/Lung_Opacity-1870.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1870.png,Lung_Opacity/masks/Lung_Opacity-1870.png,True,False,299,299,L,151.64,4a23959e9a3a638b351c96fae5cff594
Lung_Opacity-1871.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1871.png,Lung_Opacity/images/Lung_Opacity-1871.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1871.png,Lung_Opacity/masks/Lung_Opacity-1871.png,True,False,299,299,L,144.79,a60e549cd037373adf829d86f0c2df7b
Lung_Opacity-1872.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1872.png,Lung_Opacity/images/Lung_Opacity-1872.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1872.png,Lung_Opacity/masks/Lung_Opacity-1872.png,True,False,299,299,L,146.62,f38d79dd1efb98c3b4c71f91e4700ae6
Lung_Opacity-1873.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1873.png,Lung_Opacity/images/Lung_Opacity-1873.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1873.png,Lung_Opacity/masks/Lung_Opacity-1873.png,True,False,299,299,L,147.01,4111a9755ad05edc75217a0b8370481c
Lung_Opacity-1874.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1874.png,Lung_Opacity/images/Lung_Opacity-1874.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1874.png,Lung_Opacity/masks/Lung_Opacity-1874.png,True,False,299,299,L,91.67,43888006f9ee226425fa948bc8d53add
Lung_Opacity-1875.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1875.png,Lung_Opacity/images/Lung_Opacity-1875.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1875.png,Lung_Opacity/masks/Lung_Opacity-1875.png,True,False,299,299,L,127.65,d18edf7b1b2bdf5642a8526cd2da1e66
Lung_Opacity-1876.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1876.png,Lung_Opacity/images/Lung_Opacity-1876.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1876.png,Lung_Opacity/masks/Lung_Opacity-1876.png,True,False,299,299,L,92.53,f62f6b1a94a2ecee5aa886129fa57431
Lung_Opacity-1877.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1877.png,Lung_Opacity/images/Lung_Opacity-1877.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1877.png,Lung_Opacity/masks/Lung_Opacity-1877.png,True,False,299,299,L,117.6,5282e61cbb0b29e6b8075ebf00cf89db
Lung_Opacity-1878.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1878.png,Lung_Opacity/images/Lung_Opacity-1878.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1878.png,Lung_Opacity/masks/Lung_Opacity-1878.png,True,False,299,299,L,119.6,286f33cfc1844ab85a7c13adeaafd0f7
Lung_Opacity-1879.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1879.png,Lung_Opacity/images/Lung_Opacity-1879.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1879.png,Lung_Opacity/masks/Lung_Opacity-1879.png,True,False,299,299,L,120.0,0f164c414e620ba7355e20c1f509474d
Lung_Opacity-188.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-188.png,Lung_Opacity/images/Lung_Opacity-188.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-188.png,Lung_Opacity/masks/Lung_Opacity-188.png,True,False,299,299,L,131.6,370f8b6b8392af29daffb7def49dc81c
Lung_Opacity-1880.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1880.png,Lung_Opacity/images/Lung_Opacity-1880.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1880.png,Lung_Opacity/masks/Lung_Opacity-1880.png,True,False,299,299,L,116.24,53206597d5a73768ddc105beebafc4cf
Lung_Opacity-1881.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1881.png,Lung_Opacity/images/Lung_Opacity-1881.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1881.png,Lung_Opacity/masks/Lung_Opacity-1881.png,True,False,299,299,L,131.13,add923f394df1b7730154d4dfdfe9421
Lung_Opacity-1882.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1882.png,Lung_Opacity/images/Lung_Opacity-1882.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1882.png,Lung_Opacity/masks/Lung_Opacity-1882.png,True,False,299,299,L,131.84,81951afb85087353fab376172506af0e
Lung_Opacity-1883.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1883.png,Lung_Opacity/images/Lung_Opacity-1883.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1883.png,Lung_Opacity/masks/Lung_Opacity-1883.png,True,False,299,299,L,108.36,57e2db5ac36860b0a2a089468b5af46b
Lung_Opacity-1884.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1884.png,Lung_Opacity/images/Lung_Opacity-1884.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1884.png,Lung_Opacity/masks/Lung_Opacity-1884.png,True,False,299,299,L,149.01,c35525300a4230ddfc6fdfacd49408e5
Lung_Opacity-1885.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1885.png,Lung_Opacity/images/Lung_Opacity-1885.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1885.png,Lung_Opacity/masks/Lung_Opacity-1885.png,True,False,299,299,L,111.99,e190d086277cb9fb6c95928322ffa75c
Lung_Opacity-1886.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1886.png,Lung_Opacity/images/Lung_Opacity-1886.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1886.png,Lung_Opacity/masks/Lung_Opacity-1886.png,True,False,299,299,L,109.75,300652fb3a934372f6f33c477f977206
Lung_Opacity-1887.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1887.png,Lung_Opacity/images/Lung_Opacity-1887.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1887.png,Lung_Opacity/masks/Lung_Opacity-1887.png,True,False,299,299,L,143.13,d2f1573c2b8376141c0c786b13b5441c
Lung_Opacity-1888.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1888.png,Lung_Opacity/images/Lung_Opacity-1888.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1888.png,Lung_Opacity/masks/Lung_Opacity-1888.png,True,False,299,299,L,137.62,10307ac84e0d35054b9374d6e80762a3
Lung_Opacity-1889.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1889.png,Lung_Opacity/images/Lung_Opacity-1889.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1889.png,Lung_Opacity/masks/Lung_Opacity-1889.png,True,False,299,299,L,133.32,4566d8f866b3ff929db4573fa577affe
Lung_Opacity-189.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-189.png,Lung_Opacity/images/Lung_Opacity-189.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-189.png,Lung_Opacity/masks/Lung_Opacity-189.png,True,False,299,299,L,105.04,a3e6df76756d463854fcb425d5836c40
Lung_Opacity-1890.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1890.png,Lung_Opacity/images/Lung_Opacity-1890.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1890.png,Lung_Opacity/masks/Lung_Opacity-1890.png,True,False,299,299,L,83.52,dea1372b92ddf78e46d461486136b47f
Lung_Opacity-1891.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1891.png,Lung_Opacity/images/Lung_Opacity-1891.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1891.png,Lung_Opacity/masks/Lung_Opacity-1891.png,True,False,299,299,L,159.16,e62018092797e659618a784dbdeec1ec
Lung_Opacity-1892.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1892.png,Lung_Opacity/images/Lung_Opacity-1892.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1892.png,Lung_Opacity/masks/Lung_Opacity-1892.png,True,False,299,299,L,56.58,2675e36e6e300eb2b3e9c25de08b4e8c
Lung_Opacity-1893.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1893.png,Lung_Opacity/images/Lung_Opacity-1893.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1893.png,Lung_Opacity/masks/Lung_Opacity-1893.png,True,False,299,299,L,143.91,cb746e7e75de1ec582f8d14691711080
Lung_Opacity-1894.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1894.png,Lung_Opacity/images/Lung_Opacity-1894.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1894.png,Lung_Opacity/masks/Lung_Opacity-1894.png,True,False,299,299,L,120.41,11d0e6b36f1f8768e8598433cbead966
Lung_Opacity-1895.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1895.png,Lung_Opacity/images/Lung_Opacity-1895.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1895.png,Lung_Opacity/masks/Lung_Opacity-1895.png,True,False,299,299,L,133.79,3e0077075ebbf82e9ca940e75f0cd01d
Lung_Opacity-1896.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1896.png,Lung_Opacity/images/Lung_Opacity-1896.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1896.png,Lung_Opacity/masks/Lung_Opacity-1896.png,True,False,299,299,L,139.69,93f6ec5ebd8e275f70ef4bced5712647
Lung_Opacity-1897.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1897.png,Lung_Opacity/images/Lung_Opacity-1897.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1897.png,Lung_Opacity/masks/Lung_Opacity-1897.png,True,False,299,299,L,127.28,a8f529cb83cfd24caf74d2e3cdaf6d57
Lung_Opacity-1898.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1898.png,Lung_Opacity/images/Lung_Opacity-1898.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1898.png,Lung_Opacity/masks/Lung_Opacity-1898.png,True,False,299,299,L,116.88,fa7932ee43cadc0ef2e54d6533a4e6e9
Lung_Opacity-1899.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1899.png,Lung_Opacity/images/Lung_Opacity-1899.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1899.png,Lung_Opacity/masks/Lung_Opacity-1899.png,True,False,299,299,L,110.95,eda7cdba4e3e67e2d37f35d695b8f806
Lung_Opacity-19.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-19.png,Lung_Opacity/images/Lung_Opacity-19.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-19.png,Lung_Opacity/masks/Lung_Opacity-19.png,True,False,299,299,L,94.85,cfac9df791266a4622b798f8b30d7364
Lung_Opacity-190.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-190.png,Lung_Opacity/images/Lung_Opacity-190.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-190.png,Lung_Opacity/masks/Lung_Opacity-190.png,True,False,299,299,L,117.3,b69d2f4b2e52d8fcdc218e5134440ff0
Lung_Opacity-1900.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1900.png,Lung_Opacity/images/Lung_Opacity-1900.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1900.png,Lung_Opacity/masks/Lung_Opacity-1900.png,True,False,299,299,L,112.28,6f379256374f786424f18adedd9591df
Lung_Opacity-1901.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1901.png,Lung_Opacity/images/Lung_Opacity-1901.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1901.png,Lung_Opacity/masks/Lung_Opacity-1901.png,True,False,299,299,L,153.39,6c79e2b7adf005f7ac7b4c7a06b31ed6
Lung_Opacity-1902.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1902.png,Lung_Opacity/images/Lung_Opacity-1902.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1902.png,Lung_Opacity/masks/Lung_Opacity-1902.png,True,False,299,299,L,116.46,04d80526ce7a13f6fc028c68cb5efe5a
Lung_Opacity-1903.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1903.png,Lung_Opacity/images/Lung_Opacity-1903.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1903.png,Lung_Opacity/masks/Lung_Opacity-1903.png,True,False,299,299,L,117.08,09efa95c8d93b26a4d8abaa6713e60ea
Lung_Opacity-1904.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1904.png,Lung_Opacity/images/Lung_Opacity-1904.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1904.png,Lung_Opacity/masks/Lung_Opacity-1904.png,True,False,299,299,L,184.32,bd6b51fbfc499382ad5ff7734f978e09
Lung_Opacity-1905.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1905.png,Lung_Opacity/images/Lung_Opacity-1905.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1905.png,Lung_Opacity/masks/Lung_Opacity-1905.png,True,False,299,299,L,127.38,1945e2256908a81e86852a6c06d1646c
Lung_Opacity-1906.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1906.png,Lung_Opacity/images/Lung_Opacity-1906.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1906.png,Lung_Opacity/masks/Lung_Opacity-1906.png,True,False,299,299,L,114.78,f94ae94ad30bc90e44a4fc78a3e0029a
Lung_Opacity-1907.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1907.png,Lung_Opacity/images/Lung_Opacity-1907.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1907.png,Lung_Opacity/masks/Lung_Opacity-1907.png,True,False,299,299,L,115.47,e71afd01239ec5094a9e642a49ebc171
Lung_Opacity-1908.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1908.png,Lung_Opacity/images/Lung_Opacity-1908.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1908.png,Lung_Opacity/masks/Lung_Opacity-1908.png,True,False,299,299,L,108.1,129a5303abe2f0d23feb113f462325f7
Lung_Opacity-1909.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1909.png,Lung_Opacity/images/Lung_Opacity-1909.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1909.png,Lung_Opacity/masks/Lung_Opacity-1909.png,True,False,299,299,L,128.98,f564e90e41e48f3f6e5284f4a3e291eb
Lung_Opacity-191.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-191.png,Lung_Opacity/images/Lung_Opacity-191.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-191.png,Lung_Opacity/masks/Lung_Opacity-191.png,True,False,299,299,L,108.5,821815e72ea30c1cf15abf03e5866d5a
Lung_Opacity-1910.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1910.png,Lung_Opacity/images/Lung_Opacity-1910.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1910.png,Lung_Opacity/masks/Lung_Opacity-1910.png,True,False,299,299,L,91.74,e71e69b0386a7a5b0097cd152be6c513
Lung_Opacity-1911.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1911.png,Lung_Opacity/images/Lung_Opacity-1911.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1911.png,Lung_Opacity/masks/Lung_Opacity-1911.png,True,False,299,299,L,154.81,f47c82561db2ff5a91aa245cf23cfa94
Lung_Opacity-1912.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1912.png,Lung_Opacity/images/Lung_Opacity-1912.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1912.png,Lung_Opacity/masks/Lung_Opacity-1912.png,True,False,299,299,L,162.12,f9d13885abbd11028fd3d46b7ff79adb
Lung_Opacity-1913.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1913.png,Lung_Opacity/images/Lung_Opacity-1913.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1913.png,Lung_Opacity/masks/Lung_Opacity-1913.png,True,False,299,299,L,103.42,7d6cb7cab33716db3ce6837e0bbdf032
Lung_Opacity-1914.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1914.png,Lung_Opacity/images/Lung_Opacity-1914.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1914.png,Lung_Opacity/masks/Lung_Opacity-1914.png,True,False,299,299,L,105.61,d0851510ffe10dc7d027a685aee1fd10
Lung_Opacity-1915.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1915.png,Lung_Opacity/images/Lung_Opacity-1915.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1915.png,Lung_Opacity/masks/Lung_Opacity-1915.png,True,False,299,299,L,86.47,6f46a31a439ad16a17e30d5375acd324
Lung_Opacity-1916.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1916.png,Lung_Opacity/images/Lung_Opacity-1916.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1916.png,Lung_Opacity/masks/Lung_Opacity-1916.png,True,False,299,299,L,111.2,600fe80f56311edeb0fa99712e2eeb4a
Lung_Opacity-1917.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1917.png,Lung_Opacity/images/Lung_Opacity-1917.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1917.png,Lung_Opacity/masks/Lung_Opacity-1917.png,True,False,299,299,L,109.68,a919caa13be8696aab46907018334b24
Lung_Opacity-1918.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1918.png,Lung_Opacity/images/Lung_Opacity-1918.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1918.png,Lung_Opacity/masks/Lung_Opacity-1918.png,True,False,299,299,L,148.5,f3e4e923d5aaf7c665248ce15bdea94d
Lung_Opacity-1919.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1919.png,Lung_Opacity/images/Lung_Opacity-1919.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1919.png,Lung_Opacity/masks/Lung_Opacity-1919.png,True,False,299,299,L,114.72,7f6096bb158509bc15156c8f1dfd4425
Lung_Opacity-192.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-192.png,Lung_Opacity/images/Lung_Opacity-192.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-192.png,Lung_Opacity/masks/Lung_Opacity-192.png,True,False,299,299,L,125.08,9b0a8d64f87ee898a60d0b9fa2120ae8
Lung_Opacity-1920.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1920.png,Lung_Opacity/images/Lung_Opacity-1920.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1920.png,Lung_Opacity/masks/Lung_Opacity-1920.png,True,False,299,299,L,131.7,e73ae88b1eba016772656799014f788f
Lung_Opacity-1921.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1921.png,Lung_Opacity/images/Lung_Opacity-1921.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1921.png,Lung_Opacity/masks/Lung_Opacity-1921.png,True,False,299,299,L,115.76,2f0566e84ae6adace7de88007de06932
Lung_Opacity-1922.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1922.png,Lung_Opacity/images/Lung_Opacity-1922.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1922.png,Lung_Opacity/masks/Lung_Opacity-1922.png,True,False,299,299,L,169.47,b81f9b206ec38cbabcf8b37363c4461b
Lung_Opacity-1923.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1923.png,Lung_Opacity/images/Lung_Opacity-1923.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1923.png,Lung_Opacity/masks/Lung_Opacity-1923.png,True,False,299,299,L,109.97,ff2fe24eae72cd0acde3dc0398deaad8
Lung_Opacity-1924.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1924.png,Lung_Opacity/images/Lung_Opacity-1924.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1924.png,Lung_Opacity/masks/Lung_Opacity-1924.png,True,False,299,299,L,150.55,90bcbd668f133cc6284877dac333817d
Lung_Opacity-1925.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1925.png,Lung_Opacity/images/Lung_Opacity-1925.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1925.png,Lung_Opacity/masks/Lung_Opacity-1925.png,True,False,299,299,L,128.99,07fc2739d2b2da4af0dd3b064cbe0cb0
Lung_Opacity-1926.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1926.png,Lung_Opacity/images/Lung_Opacity-1926.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1926.png,Lung_Opacity/masks/Lung_Opacity-1926.png,True,False,299,299,L,124.69,a5a571083b0a453fe02790a3458afa70
Lung_Opacity-1927.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1927.png,Lung_Opacity/images/Lung_Opacity-1927.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1927.png,Lung_Opacity/masks/Lung_Opacity-1927.png,True,False,299,299,L,115.14,de9e7cde0db9afc8915f4cf48afb0f32
Lung_Opacity-1928.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1928.png,Lung_Opacity/images/Lung_Opacity-1928.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1928.png,Lung_Opacity/masks/Lung_Opacity-1928.png,True,False,299,299,L,124.99,816d9199dfe87364d45e3ddbcaf85467
Lung_Opacity-1929.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1929.png,Lung_Opacity/images/Lung_Opacity-1929.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1929.png,Lung_Opacity/masks/Lung_Opacity-1929.png,True,False,299,299,L,96.73,d67877d207aec91654c5fce75f4e7022
Lung_Opacity-193.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-193.png,Lung_Opacity/images/Lung_Opacity-193.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-193.png,Lung_Opacity/masks/Lung_Opacity-193.png,True,False,299,299,L,97.94,79d383b93c7660821646b721d8533da2
Lung_Opacity-1930.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1930.png,Lung_Opacity/images/Lung_Opacity-1930.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1930.png,Lung_Opacity/masks/Lung_Opacity-1930.png,True,False,299,299,L,114.06,781e74170a766a396a47a1b79c7e16c0
Lung_Opacity-1931.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1931.png,Lung_Opacity/images/Lung_Opacity-1931.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1931.png,Lung_Opacity/masks/Lung_Opacity-1931.png,True,False,299,299,L,146.94,9dbe6771c273647f9ab601c08cae95e3
Lung_Opacity-1932.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1932.png,Lung_Opacity/images/Lung_Opacity-1932.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1932.png,Lung_Opacity/masks/Lung_Opacity-1932.png,True,False,299,299,L,92.72,31550a4bd98d57e8047e4fac7eb49771
Lung_Opacity-1933.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1933.png,Lung_Opacity/images/Lung_Opacity-1933.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1933.png,Lung_Opacity/masks/Lung_Opacity-1933.png,True,False,299,299,L,115.32,0edb2c79ba79980ee7b554ce8be9696f
Lung_Opacity-1934.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1934.png,Lung_Opacity/images/Lung_Opacity-1934.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1934.png,Lung_Opacity/masks/Lung_Opacity-1934.png,True,False,299,299,L,113.66,f9b38596309a54f39629e786e5248177
Lung_Opacity-1935.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1935.png,Lung_Opacity/images/Lung_Opacity-1935.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1935.png,Lung_Opacity/masks/Lung_Opacity-1935.png,True,False,299,299,L,130.04,b89d787f8dbacc79bbdc47f3b45269b2
Lung_Opacity-1936.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1936.png,Lung_Opacity/images/Lung_Opacity-1936.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1936.png,Lung_Opacity/masks/Lung_Opacity-1936.png,True,False,299,299,L,162.78,26c0c1393b931f55b919f549e6c3edc9
Lung_Opacity-1937.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1937.png,Lung_Opacity/images/Lung_Opacity-1937.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1937.png,Lung_Opacity/masks/Lung_Opacity-1937.png,True,False,299,299,L,108.33,32bb560755b65ee2d3d3f576c0883dd6
Lung_Opacity-1938.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1938.png,Lung_Opacity/images/Lung_Opacity-1938.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1938.png,Lung_Opacity/masks/Lung_Opacity-1938.png,True,False,299,299,L,160.13,9406bac43adfe920113145a76cad0883
Lung_Opacity-1939.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1939.png,Lung_Opacity/images/Lung_Opacity-1939.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1939.png,Lung_Opacity/masks/Lung_Opacity-1939.png,True,False,299,299,L,128.56,5f95396403a13cbe71de08105e66403f
Lung_Opacity-194.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-194.png,Lung_Opacity/images/Lung_Opacity-194.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-194.png,Lung_Opacity/masks/Lung_Opacity-194.png,True,False,299,299,L,115.9,caf6ac4c5588af5770374e42ab609444
Lung_Opacity-1940.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1940.png,Lung_Opacity/images/Lung_Opacity-1940.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1940.png,Lung_Opacity/masks/Lung_Opacity-1940.png,True,False,299,299,L,108.97,f9c5312eec9a298afc067cb64a2ea6cd
Lung_Opacity-1941.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1941.png,Lung_Opacity/images/Lung_Opacity-1941.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1941.png,Lung_Opacity/masks/Lung_Opacity-1941.png,True,False,299,299,L,109.7,8d25f104fd27b09e85ea514b3f332dca
Lung_Opacity-1942.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1942.png,Lung_Opacity/images/Lung_Opacity-1942.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1942.png,Lung_Opacity/masks/Lung_Opacity-1942.png,True,False,299,299,L,123.73,5a28eee676865f6e0b713b6659bfb6cb
Lung_Opacity-1943.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1943.png,Lung_Opacity/images/Lung_Opacity-1943.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1943.png,Lung_Opacity/masks/Lung_Opacity-1943.png,True,False,299,299,L,144.58,71566762b0d196178d3c5e6d91a4c7e0
Lung_Opacity-1944.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1944.png,Lung_Opacity/images/Lung_Opacity-1944.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1944.png,Lung_Opacity/masks/Lung_Opacity-1944.png,True,False,299,299,L,166.99,f5947ad7118b4579a303b594c258f174
Lung_Opacity-1945.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1945.png,Lung_Opacity/images/Lung_Opacity-1945.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1945.png,Lung_Opacity/masks/Lung_Opacity-1945.png,True,False,299,299,L,135.57,bcb13d5b36233df0b2294b1d472d8e8a
Lung_Opacity-1946.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1946.png,Lung_Opacity/images/Lung_Opacity-1946.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1946.png,Lung_Opacity/masks/Lung_Opacity-1946.png,True,False,299,299,L,113.31,00030c4aedde54573684165d26089bc1
Lung_Opacity-1947.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1947.png,Lung_Opacity/images/Lung_Opacity-1947.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1947.png,Lung_Opacity/masks/Lung_Opacity-1947.png,True,False,299,299,L,115.3,fb7adeecd1f4e2d0187c46ae008840d0
Lung_Opacity-1948.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1948.png,Lung_Opacity/images/Lung_Opacity-1948.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1948.png,Lung_Opacity/masks/Lung_Opacity-1948.png,True,False,299,299,L,145.4,2379b543d256ff26240f8baf7989a600
Lung_Opacity-1949.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1949.png,Lung_Opacity/images/Lung_Opacity-1949.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1949.png,Lung_Opacity/masks/Lung_Opacity-1949.png,True,False,299,299,L,138.18,ef6fd26fad934c3f9173ab269977f148
Lung_Opacity-195.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-195.png,Lung_Opacity/images/Lung_Opacity-195.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-195.png,Lung_Opacity/masks/Lung_Opacity-195.png,True,False,299,299,L,153.48,4bdfeaaf6f6dbcd33b0b95b3bd4d4dd3
Lung_Opacity-1950.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1950.png,Lung_Opacity/images/Lung_Opacity-1950.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1950.png,Lung_Opacity/masks/Lung_Opacity-1950.png,True,False,299,299,L,110.22,386dab21b900b5da515fa3eb3f597e0e
Lung_Opacity-1951.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1951.png,Lung_Opacity/images/Lung_Opacity-1951.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1951.png,Lung_Opacity/masks/Lung_Opacity-1951.png,True,False,299,299,L,120.63,1ede51d2c54ab27ca5bf4f89e20a3861
Lung_Opacity-1952.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1952.png,Lung_Opacity/images/Lung_Opacity-1952.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1952.png,Lung_Opacity/masks/Lung_Opacity-1952.png,True,False,299,299,L,158.64,d3261984d78045a84e49c8c13ed40c43
Lung_Opacity-1953.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1953.png,Lung_Opacity/images/Lung_Opacity-1953.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1953.png,Lung_Opacity/masks/Lung_Opacity-1953.png,True,False,299,299,L,79.76,a938393511f00d01a698076358b1b1b7
Lung_Opacity-1954.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1954.png,Lung_Opacity/images/Lung_Opacity-1954.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1954.png,Lung_Opacity/masks/Lung_Opacity-1954.png,True,False,299,299,L,140.17,84688897a51e248844015683409858c3
Lung_Opacity-1955.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1955.png,Lung_Opacity/images/Lung_Opacity-1955.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1955.png,Lung_Opacity/masks/Lung_Opacity-1955.png,True,False,299,299,L,140.46,f626c6e6464bb9872831bef247897d41
Lung_Opacity-1956.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1956.png,Lung_Opacity/images/Lung_Opacity-1956.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1956.png,Lung_Opacity/masks/Lung_Opacity-1956.png,True,False,299,299,L,124.35,5928c76b78c68a9bd730c84215a8b0fc
Lung_Opacity-1957.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1957.png,Lung_Opacity/images/Lung_Opacity-1957.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1957.png,Lung_Opacity/masks/Lung_Opacity-1957.png,True,False,299,299,L,153.09,1d841e18672fffd578b78a5de8ef3cab
Lung_Opacity-1958.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1958.png,Lung_Opacity/images/Lung_Opacity-1958.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1958.png,Lung_Opacity/masks/Lung_Opacity-1958.png,True,False,299,299,L,162.83,029dc5a4aebfd61415d5d43a3919cab2
Lung_Opacity-1959.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1959.png,Lung_Opacity/images/Lung_Opacity-1959.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1959.png,Lung_Opacity/masks/Lung_Opacity-1959.png,True,False,299,299,L,139.33,bdf04f9e4f1f76249ea118d928119abe
Lung_Opacity-196.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-196.png,Lung_Opacity/images/Lung_Opacity-196.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-196.png,Lung_Opacity/masks/Lung_Opacity-196.png,True,False,299,299,L,140.39,fca8de2a3fd99d574c9d11948a86167e
Lung_Opacity-1960.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1960.png,Lung_Opacity/images/Lung_Opacity-1960.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1960.png,Lung_Opacity/masks/Lung_Opacity-1960.png,True,False,299,299,L,106.76,2667d4dc83227690976a4736e1db99f8
Lung_Opacity-1961.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1961.png,Lung_Opacity/images/Lung_Opacity-1961.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1961.png,Lung_Opacity/masks/Lung_Opacity-1961.png,True,False,299,299,L,88.19,d1b3690b2cc01de6c8f73259ef7269cf
Lung_Opacity-1962.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1962.png,Lung_Opacity/images/Lung_Opacity-1962.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1962.png,Lung_Opacity/masks/Lung_Opacity-1962.png,True,False,299,299,L,112.23,6bfcdfb763f70d906d08f689f8d42f23
Lung_Opacity-1963.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1963.png,Lung_Opacity/images/Lung_Opacity-1963.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1963.png,Lung_Opacity/masks/Lung_Opacity-1963.png,True,False,299,299,L,122.1,1c1f03f8f2b716e41589ae236bf52c73
Lung_Opacity-1964.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1964.png,Lung_Opacity/images/Lung_Opacity-1964.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1964.png,Lung_Opacity/masks/Lung_Opacity-1964.png,True,False,299,299,L,91.57,ae3dbc7eb7bc379d2f7e36a7950668a7
Lung_Opacity-1965.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1965.png,Lung_Opacity/images/Lung_Opacity-1965.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1965.png,Lung_Opacity/masks/Lung_Opacity-1965.png,True,False,299,299,L,118.51,be95b66b9c9be64f12943d8ae50aac61
Lung_Opacity-1966.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1966.png,Lung_Opacity/images/Lung_Opacity-1966.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1966.png,Lung_Opacity/masks/Lung_Opacity-1966.png,True,False,299,299,L,164.39,44b45c95b74f8b1ef9e47133ba178aa7
Lung_Opacity-1967.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1967.png,Lung_Opacity/images/Lung_Opacity-1967.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1967.png,Lung_Opacity/masks/Lung_Opacity-1967.png,True,False,299,299,L,175.9,b1034ad4d115bd96a669fd1aecf8c1a6
Lung_Opacity-1968.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1968.png,Lung_Opacity/images/Lung_Opacity-1968.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1968.png,Lung_Opacity/masks/Lung_Opacity-1968.png,True,False,299,299,L,123.13,c8fe2b3b158edf34b3d0cd753556c454
Lung_Opacity-1969.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1969.png,Lung_Opacity/images/Lung_Opacity-1969.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1969.png,Lung_Opacity/masks/Lung_Opacity-1969.png,True,False,299,299,L,99.4,f835782d82f7fddb123368e09a7b37f5
Lung_Opacity-197.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-197.png,Lung_Opacity/images/Lung_Opacity-197.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-197.png,Lung_Opacity/masks/Lung_Opacity-197.png,True,False,299,299,L,109.47,15b2a21b1192d08be5d80ae383f2e98c
Lung_Opacity-1970.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1970.png,Lung_Opacity/images/Lung_Opacity-1970.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1970.png,Lung_Opacity/masks/Lung_Opacity-1970.png,True,False,299,299,L,166.51,799634ebe3a85bb49634eaee211b5b4c
Lung_Opacity-1971.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1971.png,Lung_Opacity/images/Lung_Opacity-1971.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1971.png,Lung_Opacity/masks/Lung_Opacity-1971.png,True,False,299,299,L,111.36,5c6ab3e7cfb8679692fb542afeca200d
Lung_Opacity-1972.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1972.png,Lung_Opacity/images/Lung_Opacity-1972.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1972.png,Lung_Opacity/masks/Lung_Opacity-1972.png,True,False,299,299,L,101.77,0beddaa09c7ed8cec3f7c3c5fc6096ba
Lung_Opacity-1973.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1973.png,Lung_Opacity/images/Lung_Opacity-1973.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1973.png,Lung_Opacity/masks/Lung_Opacity-1973.png,True,False,299,299,L,98.47,8c5e0c4becbc558537539803e1e6190f
Lung_Opacity-1974.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1974.png,Lung_Opacity/images/Lung_Opacity-1974.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1974.png,Lung_Opacity/masks/Lung_Opacity-1974.png,True,False,299,299,L,100.69,261f61db5ab4a2d55212e9d6ede50985
Lung_Opacity-1975.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1975.png,Lung_Opacity/images/Lung_Opacity-1975.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1975.png,Lung_Opacity/masks/Lung_Opacity-1975.png,True,False,299,299,L,114.28,7a57de128b40ced1db9b05b659c931b9
Lung_Opacity-1976.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1976.png,Lung_Opacity/images/Lung_Opacity-1976.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1976.png,Lung_Opacity/masks/Lung_Opacity-1976.png,True,False,299,299,L,156.13,4cb2a85b592459f17abc92884471e19c
Lung_Opacity-1977.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1977.png,Lung_Opacity/images/Lung_Opacity-1977.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1977.png,Lung_Opacity/masks/Lung_Opacity-1977.png,True,False,299,299,L,138.48,b709998280b29edf8f8565f5fcb42f6b
Lung_Opacity-1978.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1978.png,Lung_Opacity/images/Lung_Opacity-1978.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1978.png,Lung_Opacity/masks/Lung_Opacity-1978.png,True,False,299,299,L,137.83,c4c92b94e0d9f563ed49ba13ae9afffd
Lung_Opacity-1979.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1979.png,Lung_Opacity/images/Lung_Opacity-1979.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1979.png,Lung_Opacity/masks/Lung_Opacity-1979.png,True,False,299,299,L,132.64,aca5dd1b5e885ef6bc669aedf04e0275
Lung_Opacity-198.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-198.png,Lung_Opacity/images/Lung_Opacity-198.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-198.png,Lung_Opacity/masks/Lung_Opacity-198.png,True,False,299,299,L,64.13,4cbd317442fc1444daf985af0aa7541c
Lung_Opacity-1980.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1980.png,Lung_Opacity/images/Lung_Opacity-1980.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1980.png,Lung_Opacity/masks/Lung_Opacity-1980.png,True,False,299,299,L,101.95,15285d6c6522b47ae3bc50d2a00b1804
Lung_Opacity-1981.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1981.png,Lung_Opacity/images/Lung_Opacity-1981.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1981.png,Lung_Opacity/masks/Lung_Opacity-1981.png,True,False,299,299,L,109.2,9c9b7c7a89a796eb526c91899f8f4aff
Lung_Opacity-1982.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1982.png,Lung_Opacity/images/Lung_Opacity-1982.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1982.png,Lung_Opacity/masks/Lung_Opacity-1982.png,True,False,299,299,L,158.45,c2591b7e8aa6be49cd256d4060e1069e
Lung_Opacity-1983.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1983.png,Lung_Opacity/images/Lung_Opacity-1983.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1983.png,Lung_Opacity/masks/Lung_Opacity-1983.png,True,False,299,299,L,121.77,e26d3195407ff9ff1f8f6ab7202e7d28
Lung_Opacity-1984.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1984.png,Lung_Opacity/images/Lung_Opacity-1984.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1984.png,Lung_Opacity/masks/Lung_Opacity-1984.png,True,False,299,299,L,110.46,f7d05830fac3e4224e5fe17211f7d931
Lung_Opacity-1985.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1985.png,Lung_Opacity/images/Lung_Opacity-1985.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1985.png,Lung_Opacity/masks/Lung_Opacity-1985.png,True,False,299,299,L,145.52,82279d79ebd9652581826f322b3e1570
Lung_Opacity-1986.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1986.png,Lung_Opacity/images/Lung_Opacity-1986.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1986.png,Lung_Opacity/masks/Lung_Opacity-1986.png,True,False,299,299,L,156.3,efe1e4b9779e64daeb3bec97b7b3c68a
Lung_Opacity-1987.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1987.png,Lung_Opacity/images/Lung_Opacity-1987.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1987.png,Lung_Opacity/masks/Lung_Opacity-1987.png,True,False,299,299,L,145.28,15dc43d1b5f2cd1e57ac66d88b9f7df5
Lung_Opacity-1988.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1988.png,Lung_Opacity/images/Lung_Opacity-1988.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1988.png,Lung_Opacity/masks/Lung_Opacity-1988.png,True,False,299,299,L,129.16,52249161f8af4d536d35badf92ab225d
Lung_Opacity-1989.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1989.png,Lung_Opacity/images/Lung_Opacity-1989.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1989.png,Lung_Opacity/masks/Lung_Opacity-1989.png,True,False,299,299,L,144.02,4820ba706c8a24c5faeae5e66786d04a
Lung_Opacity-199.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-199.png,Lung_Opacity/images/Lung_Opacity-199.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-199.png,Lung_Opacity/masks/Lung_Opacity-199.png,True,False,299,299,L,134.35,b64f7b6a1ac3582b281921557befc7c3
Lung_Opacity-1990.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1990.png,Lung_Opacity/images/Lung_Opacity-1990.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1990.png,Lung_Opacity/masks/Lung_Opacity-1990.png,True,False,299,299,L,113.55,ba876ba65c00d36b89cb4895682da93d
Lung_Opacity-1991.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1991.png,Lung_Opacity/images/Lung_Opacity-1991.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1991.png,Lung_Opacity/masks/Lung_Opacity-1991.png,True,False,299,299,L,109.07,9ea8d7f7dc2b9edd12d6ef3c49e77e12
Lung_Opacity-1992.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1992.png,Lung_Opacity/images/Lung_Opacity-1992.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1992.png,Lung_Opacity/masks/Lung_Opacity-1992.png,True,False,299,299,L,138.2,5e7a0c27adbebcc0b2bde18c0a260886
Lung_Opacity-1993.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1993.png,Lung_Opacity/images/Lung_Opacity-1993.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1993.png,Lung_Opacity/masks/Lung_Opacity-1993.png,True,False,299,299,L,146.44,d8945efbde59c30e202cafaf3ba989a1
Lung_Opacity-1994.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1994.png,Lung_Opacity/images/Lung_Opacity-1994.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1994.png,Lung_Opacity/masks/Lung_Opacity-1994.png,True,False,299,299,L,46.81,98e9b09d3ae56778a2f65d5c582932b1
Lung_Opacity-1995.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1995.png,Lung_Opacity/images/Lung_Opacity-1995.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1995.png,Lung_Opacity/masks/Lung_Opacity-1995.png,True,False,299,299,L,129.16,6cb8aebd5f368e50371fa7190ae73e70
Lung_Opacity-1996.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1996.png,Lung_Opacity/images/Lung_Opacity-1996.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1996.png,Lung_Opacity/masks/Lung_Opacity-1996.png,True,False,299,299,L,109.42,f4d0ffe4281598e74c798374a061ee69
Lung_Opacity-1997.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1997.png,Lung_Opacity/images/Lung_Opacity-1997.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1997.png,Lung_Opacity/masks/Lung_Opacity-1997.png,True,False,299,299,L,147.82,7a0b3a0f41dadb53f7bb496c5b1c1eaf
Lung_Opacity-1998.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1998.png,Lung_Opacity/images/Lung_Opacity-1998.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1998.png,Lung_Opacity/masks/Lung_Opacity-1998.png,True,False,299,299,L,159.14,688b3d3672dc2d3877558fc0959feece
Lung_Opacity-1999.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-1999.png,Lung_Opacity/images/Lung_Opacity-1999.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-1999.png,Lung_Opacity/masks/Lung_Opacity-1999.png,True,False,299,299,L,162.99,cd6461726fb3a00d4bd66d4153bc5b3d
Lung_Opacity-2.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2.png,Lung_Opacity/images/Lung_Opacity-2.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2.png,Lung_Opacity/masks/Lung_Opacity-2.png,True,False,299,299,L,106.0,8dc48e422c049a919ad1f7f2c73da6b1
Lung_Opacity-20.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-20.png,Lung_Opacity/images/Lung_Opacity-20.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-20.png,Lung_Opacity/masks/Lung_Opacity-20.png,True,False,299,299,L,105.41,266f501546698b14f90dff4abb7f9771
Lung_Opacity-200.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-200.png,Lung_Opacity/images/Lung_Opacity-200.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-200.png,Lung_Opacity/masks/Lung_Opacity-200.png,True,False,299,299,L,90.4,90219141c06b769326fc40f7cdec5350
Lung_Opacity-2000.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2000.png,Lung_Opacity/images/Lung_Opacity-2000.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2000.png,Lung_Opacity/masks/Lung_Opacity-2000.png,True,False,299,299,L,113.23,f984c8cd0b77131b4826594a63120dd4
Lung_Opacity-2001.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2001.png,Lung_Opacity/images/Lung_Opacity-2001.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2001.png,Lung_Opacity/masks/Lung_Opacity-2001.png,True,False,299,299,L,161.34,1be36a6e059a91f88dc8456fa39d8d30
Lung_Opacity-2002.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2002.png,Lung_Opacity/images/Lung_Opacity-2002.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2002.png,Lung_Opacity/masks/Lung_Opacity-2002.png,True,False,299,299,L,125.26,f8b89c145dba11f801000f41caa3b509
Lung_Opacity-2003.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2003.png,Lung_Opacity/images/Lung_Opacity-2003.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2003.png,Lung_Opacity/masks/Lung_Opacity-2003.png,True,False,299,299,L,161.13,17e310bf0d9a0184b6e514f6db2ae8c1
Lung_Opacity-2004.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2004.png,Lung_Opacity/images/Lung_Opacity-2004.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2004.png,Lung_Opacity/masks/Lung_Opacity-2004.png,True,False,299,299,L,114.44,a0ee27ef11b4fc1426e904f10f66318c
Lung_Opacity-2005.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2005.png,Lung_Opacity/images/Lung_Opacity-2005.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2005.png,Lung_Opacity/masks/Lung_Opacity-2005.png,True,False,299,299,L,57.24,0c2bde7cd535ac4d298d3ce4b9607f49
Lung_Opacity-2006.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2006.png,Lung_Opacity/images/Lung_Opacity-2006.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2006.png,Lung_Opacity/masks/Lung_Opacity-2006.png,True,False,299,299,L,118.1,d4e6f9ffba41f182b081384fc0c74595
Lung_Opacity-2007.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2007.png,Lung_Opacity/images/Lung_Opacity-2007.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2007.png,Lung_Opacity/masks/Lung_Opacity-2007.png,True,False,299,299,L,136.29,20b36ba17daff1d12ae08b7968baa781
Lung_Opacity-2008.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2008.png,Lung_Opacity/images/Lung_Opacity-2008.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2008.png,Lung_Opacity/masks/Lung_Opacity-2008.png,True,False,299,299,L,108.23,7cfd1c463e1e075c7e55b32a1eaebe05
Lung_Opacity-2009.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2009.png,Lung_Opacity/images/Lung_Opacity-2009.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2009.png,Lung_Opacity/masks/Lung_Opacity-2009.png,True,False,299,299,L,103.25,c4413e1c802698069f63118b1dcc4905
Lung_Opacity-201.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-201.png,Lung_Opacity/images/Lung_Opacity-201.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-201.png,Lung_Opacity/masks/Lung_Opacity-201.png,True,False,299,299,L,132.01,1cf527863a76cf704ddad84e0fe7ea1a
Lung_Opacity-2010.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2010.png,Lung_Opacity/images/Lung_Opacity-2010.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2010.png,Lung_Opacity/masks/Lung_Opacity-2010.png,True,False,299,299,L,118.87,f71d3d061349db04752cce63eca6214e
Lung_Opacity-2011.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2011.png,Lung_Opacity/images/Lung_Opacity-2011.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2011.png,Lung_Opacity/masks/Lung_Opacity-2011.png,True,False,299,299,L,112.06,a92e311a1b9d49a2441e23da4a8a95d9
Lung_Opacity-2012.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2012.png,Lung_Opacity/images/Lung_Opacity-2012.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2012.png,Lung_Opacity/masks/Lung_Opacity-2012.png,True,False,299,299,L,121.42,0e04f9f237f3561fcd65fb0a89313eb7
Lung_Opacity-2013.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2013.png,Lung_Opacity/images/Lung_Opacity-2013.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2013.png,Lung_Opacity/masks/Lung_Opacity-2013.png,True,False,299,299,L,84.29,84c93a12f520ccccc5152a91e83c3460
Lung_Opacity-2014.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2014.png,Lung_Opacity/images/Lung_Opacity-2014.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2014.png,Lung_Opacity/masks/Lung_Opacity-2014.png,True,False,299,299,L,177.37,13d86753b1f12e79fabe763814edb157
Lung_Opacity-2015.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2015.png,Lung_Opacity/images/Lung_Opacity-2015.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2015.png,Lung_Opacity/masks/Lung_Opacity-2015.png,True,False,299,299,L,135.5,9ec27ddd4cb91cf6b5750f53c45178f1
Lung_Opacity-2016.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2016.png,Lung_Opacity/images/Lung_Opacity-2016.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2016.png,Lung_Opacity/masks/Lung_Opacity-2016.png,True,False,299,299,L,133.87,cdf0c1d5d8ef0b21d8044b79a6998bab
Lung_Opacity-2017.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2017.png,Lung_Opacity/images/Lung_Opacity-2017.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2017.png,Lung_Opacity/masks/Lung_Opacity-2017.png,True,False,299,299,L,110.9,00fda914ba1ccd34ca1f0cf07dd7e4fa
Lung_Opacity-2018.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2018.png,Lung_Opacity/images/Lung_Opacity-2018.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2018.png,Lung_Opacity/masks/Lung_Opacity-2018.png,True,False,299,299,L,115.89,93e4ec8876b8309e16327d2b6ef3606e
Lung_Opacity-2019.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2019.png,Lung_Opacity/images/Lung_Opacity-2019.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2019.png,Lung_Opacity/masks/Lung_Opacity-2019.png,True,False,299,299,L,117.79,5ac5f2a3a4c435ee5f88f5baa7c688d1
Lung_Opacity-202.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-202.png,Lung_Opacity/images/Lung_Opacity-202.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-202.png,Lung_Opacity/masks/Lung_Opacity-202.png,True,False,299,299,L,113.8,f4fd241b0f073d5a06436a3bf6547396
Lung_Opacity-2020.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2020.png,Lung_Opacity/images/Lung_Opacity-2020.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2020.png,Lung_Opacity/masks/Lung_Opacity-2020.png,True,False,299,299,L,127.18,5c4030a503a41ff903d05def5357dbdb
Lung_Opacity-2021.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2021.png,Lung_Opacity/images/Lung_Opacity-2021.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2021.png,Lung_Opacity/masks/Lung_Opacity-2021.png,True,False,299,299,L,140.53,845c624a2f9d980f6af09f8181d9c9db
Lung_Opacity-2022.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2022.png,Lung_Opacity/images/Lung_Opacity-2022.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2022.png,Lung_Opacity/masks/Lung_Opacity-2022.png,True,False,299,299,L,131.24,9d47a153f0cd6b4081b5c3b5238fe4f1
Lung_Opacity-2023.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2023.png,Lung_Opacity/images/Lung_Opacity-2023.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2023.png,Lung_Opacity/masks/Lung_Opacity-2023.png,True,False,299,299,L,142.16,83652a8558e264741af71ac14c8aa8f2
Lung_Opacity-2024.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2024.png,Lung_Opacity/images/Lung_Opacity-2024.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2024.png,Lung_Opacity/masks/Lung_Opacity-2024.png,True,False,299,299,L,122.29,ea38f7701e577821d70384e084cd7a8d
Lung_Opacity-2025.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2025.png,Lung_Opacity/images/Lung_Opacity-2025.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2025.png,Lung_Opacity/masks/Lung_Opacity-2025.png,True,False,299,299,L,111.78,4e1c157faed963ae6903c7c74739ce03
Lung_Opacity-2026.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2026.png,Lung_Opacity/images/Lung_Opacity-2026.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2026.png,Lung_Opacity/masks/Lung_Opacity-2026.png,True,False,299,299,L,152.12,c080caa519a8aa1b8fcb69f066215867
Lung_Opacity-2027.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2027.png,Lung_Opacity/images/Lung_Opacity-2027.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2027.png,Lung_Opacity/masks/Lung_Opacity-2027.png,True,False,299,299,L,155.32,991d55b0ff4d63e6e1fd61150cb29eb5
Lung_Opacity-2028.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2028.png,Lung_Opacity/images/Lung_Opacity-2028.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2028.png,Lung_Opacity/masks/Lung_Opacity-2028.png,True,False,299,299,L,112.47,3e89d81e810cdc8ac262ed2369e73045
Lung_Opacity-2029.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2029.png,Lung_Opacity/images/Lung_Opacity-2029.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2029.png,Lung_Opacity/masks/Lung_Opacity-2029.png,True,False,299,299,L,108.28,91f9b33560c7226a41b8b71ed5528281
Lung_Opacity-203.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-203.png,Lung_Opacity/images/Lung_Opacity-203.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-203.png,Lung_Opacity/masks/Lung_Opacity-203.png,True,False,299,299,L,116.4,fedede45b58d5425b88d410e9eb0ab93
Lung_Opacity-2030.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2030.png,Lung_Opacity/images/Lung_Opacity-2030.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2030.png,Lung_Opacity/masks/Lung_Opacity-2030.png,True,False,299,299,L,144.99,fb44e11abf8b5f594e92db14c9cedbbb
Lung_Opacity-2031.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2031.png,Lung_Opacity/images/Lung_Opacity-2031.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2031.png,Lung_Opacity/masks/Lung_Opacity-2031.png,True,False,299,299,L,128.26,7d8efde94d22ff60e45cdd8bde1e0bfc
Lung_Opacity-2032.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2032.png,Lung_Opacity/images/Lung_Opacity-2032.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2032.png,Lung_Opacity/masks/Lung_Opacity-2032.png,True,False,299,299,L,103.23,c71b68895c914e7b1f78a6ce94de67c0
Lung_Opacity-2033.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2033.png,Lung_Opacity/images/Lung_Opacity-2033.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2033.png,Lung_Opacity/masks/Lung_Opacity-2033.png,True,False,299,299,L,159.78,6ae9d0d537a5a2416eef9c0387b8d21d
Lung_Opacity-2034.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2034.png,Lung_Opacity/images/Lung_Opacity-2034.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2034.png,Lung_Opacity/masks/Lung_Opacity-2034.png,True,False,299,299,L,135.07,3cdf9a9ac8b8758dc304368599c6722b
Lung_Opacity-2035.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2035.png,Lung_Opacity/images/Lung_Opacity-2035.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2035.png,Lung_Opacity/masks/Lung_Opacity-2035.png,True,False,299,299,L,116.36,88af298222ad6da4dd8ea46ad8cfb16a
Lung_Opacity-2036.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2036.png,Lung_Opacity/images/Lung_Opacity-2036.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2036.png,Lung_Opacity/masks/Lung_Opacity-2036.png,True,False,299,299,L,152.69,4a9a1528cc2dd88be9be48841cb4445e
Lung_Opacity-2037.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2037.png,Lung_Opacity/images/Lung_Opacity-2037.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2037.png,Lung_Opacity/masks/Lung_Opacity-2037.png,True,False,299,299,L,140.51,83dd786cf7334617f8b0ac0f5685494a
Lung_Opacity-2038.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2038.png,Lung_Opacity/images/Lung_Opacity-2038.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2038.png,Lung_Opacity/masks/Lung_Opacity-2038.png,True,False,299,299,L,107.51,e077e7008912329d88e547b313fa31b3
Lung_Opacity-2039.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2039.png,Lung_Opacity/images/Lung_Opacity-2039.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2039.png,Lung_Opacity/masks/Lung_Opacity-2039.png,True,False,299,299,L,109.84,263a4ed210fe7f603105c14b9cc6b752
Lung_Opacity-204.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-204.png,Lung_Opacity/images/Lung_Opacity-204.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-204.png,Lung_Opacity/masks/Lung_Opacity-204.png,True,False,299,299,L,114.27,61a7e97c9592b37c63a468b1fb0e49ae
Lung_Opacity-2040.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2040.png,Lung_Opacity/images/Lung_Opacity-2040.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2040.png,Lung_Opacity/masks/Lung_Opacity-2040.png,True,False,299,299,L,89.35,1df9f520686dc9fe28893e35971a27c5
Lung_Opacity-2041.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2041.png,Lung_Opacity/images/Lung_Opacity-2041.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2041.png,Lung_Opacity/masks/Lung_Opacity-2041.png,True,False,299,299,L,137.91,e9fcda47ce79e2a8b7d99c14ee5be3bf
Lung_Opacity-2042.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2042.png,Lung_Opacity/images/Lung_Opacity-2042.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2042.png,Lung_Opacity/masks/Lung_Opacity-2042.png,True,False,299,299,L,119.54,00b475dff0092fe8ecce6514ad87aeb5
Lung_Opacity-2043.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2043.png,Lung_Opacity/images/Lung_Opacity-2043.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2043.png,Lung_Opacity/masks/Lung_Opacity-2043.png,True,False,299,299,L,124.86,d9daab9085508ed54d4b73c940937171
Lung_Opacity-2044.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2044.png,Lung_Opacity/images/Lung_Opacity-2044.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2044.png,Lung_Opacity/masks/Lung_Opacity-2044.png,True,False,299,299,L,112.88,a21eeb1f8fb4c8de362d0fb4a41aea9f
Lung_Opacity-2045.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2045.png,Lung_Opacity/images/Lung_Opacity-2045.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2045.png,Lung_Opacity/masks/Lung_Opacity-2045.png,True,False,299,299,L,131.64,4d76046b5fbee3d21d3214cf86bdf718
Lung_Opacity-2046.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2046.png,Lung_Opacity/images/Lung_Opacity-2046.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2046.png,Lung_Opacity/masks/Lung_Opacity-2046.png,True,False,299,299,L,119.24,b7f7d265682385e19bd34d107c03252a
Lung_Opacity-2047.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2047.png,Lung_Opacity/images/Lung_Opacity-2047.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2047.png,Lung_Opacity/masks/Lung_Opacity-2047.png,True,False,299,299,L,113.95,06ffb0826dff1e57fb469f9cf4efa9d6
Lung_Opacity-2048.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2048.png,Lung_Opacity/images/Lung_Opacity-2048.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2048.png,Lung_Opacity/masks/Lung_Opacity-2048.png,True,False,299,299,L,137.4,fbcd7db750c14f735a5cc2eb02d87eec
Lung_Opacity-2049.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2049.png,Lung_Opacity/images/Lung_Opacity-2049.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2049.png,Lung_Opacity/masks/Lung_Opacity-2049.png,True,False,299,299,L,76.41,75ef636d1c34ed2aa40b910ded7254f3
Lung_Opacity-205.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-205.png,Lung_Opacity/images/Lung_Opacity-205.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-205.png,Lung_Opacity/masks/Lung_Opacity-205.png,True,False,299,299,L,121.69,81d0168a634ee2fb993bf61b30d0b1cc
Lung_Opacity-2050.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2050.png,Lung_Opacity/images/Lung_Opacity-2050.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2050.png,Lung_Opacity/masks/Lung_Opacity-2050.png,True,False,299,299,L,170.24,1c370ab32373190aabb0aad97f04ee82
Lung_Opacity-2051.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2051.png,Lung_Opacity/images/Lung_Opacity-2051.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2051.png,Lung_Opacity/masks/Lung_Opacity-2051.png,True,False,299,299,L,117.34,f1c8ff949c5c2213d4d2c8e9921c7305
Lung_Opacity-2052.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2052.png,Lung_Opacity/images/Lung_Opacity-2052.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2052.png,Lung_Opacity/masks/Lung_Opacity-2052.png,True,False,299,299,L,149.36,6b2255b68944c47d89d4dbf710963701
Lung_Opacity-2053.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2053.png,Lung_Opacity/images/Lung_Opacity-2053.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2053.png,Lung_Opacity/masks/Lung_Opacity-2053.png,True,False,299,299,L,111.83,f918547152919de9a9b178a1077006f6
Lung_Opacity-2054.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2054.png,Lung_Opacity/images/Lung_Opacity-2054.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2054.png,Lung_Opacity/masks/Lung_Opacity-2054.png,True,False,299,299,L,161.55,914abd75c8028fb8e8871c859eb4b83e
Lung_Opacity-2055.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2055.png,Lung_Opacity/images/Lung_Opacity-2055.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2055.png,Lung_Opacity/masks/Lung_Opacity-2055.png,True,False,299,299,L,130.24,5e9226f907710329eaf4dee5071b2ea1
Lung_Opacity-2056.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2056.png,Lung_Opacity/images/Lung_Opacity-2056.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2056.png,Lung_Opacity/masks/Lung_Opacity-2056.png,True,False,299,299,L,97.81,e57f63e64196c8df676efbaabd724d55
Lung_Opacity-2057.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2057.png,Lung_Opacity/images/Lung_Opacity-2057.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2057.png,Lung_Opacity/masks/Lung_Opacity-2057.png,True,False,299,299,L,108.16,a4472f5326ee4359818382c6c0aa51e4
Lung_Opacity-2058.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2058.png,Lung_Opacity/images/Lung_Opacity-2058.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2058.png,Lung_Opacity/masks/Lung_Opacity-2058.png,True,False,299,299,L,153.64,1a4308241170138f60a2d8ccee4fc367
Lung_Opacity-2059.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2059.png,Lung_Opacity/images/Lung_Opacity-2059.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2059.png,Lung_Opacity/masks/Lung_Opacity-2059.png,True,False,299,299,L,104.85,b7fb38c2ef965d5f941754b3c06b8518
Lung_Opacity-206.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-206.png,Lung_Opacity/images/Lung_Opacity-206.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-206.png,Lung_Opacity/masks/Lung_Opacity-206.png,True,False,299,299,L,105.58,c7435b613873f2a062f40ef95ea7b585
Lung_Opacity-2060.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2060.png,Lung_Opacity/images/Lung_Opacity-2060.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2060.png,Lung_Opacity/masks/Lung_Opacity-2060.png,True,False,299,299,L,148.86,1fa0c09d3613014dbce03ae25c7ebff3
Lung_Opacity-2061.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2061.png,Lung_Opacity/images/Lung_Opacity-2061.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2061.png,Lung_Opacity/masks/Lung_Opacity-2061.png,True,False,299,299,L,149.76,a310baa52d3787707667736d227efc4f
Lung_Opacity-2062.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2062.png,Lung_Opacity/images/Lung_Opacity-2062.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2062.png,Lung_Opacity/masks/Lung_Opacity-2062.png,True,False,299,299,L,114.91,3fa6f4ae1c7fc2089de9a25f5812cb80
Lung_Opacity-2063.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2063.png,Lung_Opacity/images/Lung_Opacity-2063.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2063.png,Lung_Opacity/masks/Lung_Opacity-2063.png,True,False,299,299,L,160.1,34b12b58712f3129231f8aaaa71d42fb
Lung_Opacity-2064.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2064.png,Lung_Opacity/images/Lung_Opacity-2064.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2064.png,Lung_Opacity/masks/Lung_Opacity-2064.png,True,False,299,299,L,103.12,1962735c06749d6ba7fe010c81c9e8e8
Lung_Opacity-2065.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2065.png,Lung_Opacity/images/Lung_Opacity-2065.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2065.png,Lung_Opacity/masks/Lung_Opacity-2065.png,True,False,299,299,L,152.86,ff0ae78fba6f377393df582d3c6d332a
Lung_Opacity-2066.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2066.png,Lung_Opacity/images/Lung_Opacity-2066.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2066.png,Lung_Opacity/masks/Lung_Opacity-2066.png,True,False,299,299,L,139.26,74612973bb32ee9b317dbe55a25091f1
Lung_Opacity-2067.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2067.png,Lung_Opacity/images/Lung_Opacity-2067.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2067.png,Lung_Opacity/masks/Lung_Opacity-2067.png,True,False,299,299,L,108.4,0e4f08b4167362703d3d4f9ab7cc2155
Lung_Opacity-2068.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2068.png,Lung_Opacity/images/Lung_Opacity-2068.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2068.png,Lung_Opacity/masks/Lung_Opacity-2068.png,True,False,299,299,L,139.67,c9a17ce7f1cc88fed2cc127c2f81bc42
Lung_Opacity-2069.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2069.png,Lung_Opacity/images/Lung_Opacity-2069.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2069.png,Lung_Opacity/masks/Lung_Opacity-2069.png,True,False,299,299,L,105.66,15667516d28a172bc4ee78d12349ec81
Lung_Opacity-207.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-207.png,Lung_Opacity/images/Lung_Opacity-207.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-207.png,Lung_Opacity/masks/Lung_Opacity-207.png,True,False,299,299,L,105.18,a7196466ed3f72080e05ad7114ee2939
Lung_Opacity-2070.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2070.png,Lung_Opacity/images/Lung_Opacity-2070.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2070.png,Lung_Opacity/masks/Lung_Opacity-2070.png,True,False,299,299,L,117.65,8e4fddcba494bea3017670210aad34d2
Lung_Opacity-2071.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2071.png,Lung_Opacity/images/Lung_Opacity-2071.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2071.png,Lung_Opacity/masks/Lung_Opacity-2071.png,True,False,299,299,L,141.93,bbcd3322bd5252a2cb7caf94736c0141
Lung_Opacity-2072.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2072.png,Lung_Opacity/images/Lung_Opacity-2072.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2072.png,Lung_Opacity/masks/Lung_Opacity-2072.png,True,False,299,299,L,101.73,38840214c171a2b455b2880f337b1d0d
Lung_Opacity-2073.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2073.png,Lung_Opacity/images/Lung_Opacity-2073.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2073.png,Lung_Opacity/masks/Lung_Opacity-2073.png,True,False,299,299,L,124.53,2b39d7e16675036dda505d537ab234f9
Lung_Opacity-2074.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2074.png,Lung_Opacity/images/Lung_Opacity-2074.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2074.png,Lung_Opacity/masks/Lung_Opacity-2074.png,True,False,299,299,L,118.1,a2ea9cb0be053a992f9ab8bb5a04fdf2
Lung_Opacity-2075.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2075.png,Lung_Opacity/images/Lung_Opacity-2075.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2075.png,Lung_Opacity/masks/Lung_Opacity-2075.png,True,False,299,299,L,141.66,34fafc48a643ad0ba8c8de6c98c4172e
Lung_Opacity-2076.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2076.png,Lung_Opacity/images/Lung_Opacity-2076.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2076.png,Lung_Opacity/masks/Lung_Opacity-2076.png,True,False,299,299,L,135.19,777498f758c2ab3b0f6f357ac3a39e03
Lung_Opacity-2077.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2077.png,Lung_Opacity/images/Lung_Opacity-2077.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2077.png,Lung_Opacity/masks/Lung_Opacity-2077.png,True,False,299,299,L,125.88,57a687afc7a7307f13de30fb1eaca5c5
Lung_Opacity-2078.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2078.png,Lung_Opacity/images/Lung_Opacity-2078.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2078.png,Lung_Opacity/masks/Lung_Opacity-2078.png,True,False,299,299,L,149.6,4dba32b27d8f725cdb16be6b5ca31a19
Lung_Opacity-2079.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2079.png,Lung_Opacity/images/Lung_Opacity-2079.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2079.png,Lung_Opacity/masks/Lung_Opacity-2079.png,True,False,299,299,L,79.86,9aaeaae696581deeada5b3e37b9f011e
Lung_Opacity-208.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-208.png,Lung_Opacity/images/Lung_Opacity-208.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-208.png,Lung_Opacity/masks/Lung_Opacity-208.png,True,False,299,299,L,115.25,f727386f750edf39ce103b677ba41e11
Lung_Opacity-2080.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2080.png,Lung_Opacity/images/Lung_Opacity-2080.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2080.png,Lung_Opacity/masks/Lung_Opacity-2080.png,True,False,299,299,L,131.74,18f9fb16b4134616f29e7bbf149d8f2c
Lung_Opacity-2081.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2081.png,Lung_Opacity/images/Lung_Opacity-2081.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2081.png,Lung_Opacity/masks/Lung_Opacity-2081.png,True,False,299,299,L,113.84,3b1116565a7b9df44c0c66ff4d66fce2
Lung_Opacity-2082.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2082.png,Lung_Opacity/images/Lung_Opacity-2082.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2082.png,Lung_Opacity/masks/Lung_Opacity-2082.png,True,False,299,299,L,119.49,9205eccafc2a9d0fe9a0fcd0dba19777
Lung_Opacity-2083.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2083.png,Lung_Opacity/images/Lung_Opacity-2083.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2083.png,Lung_Opacity/masks/Lung_Opacity-2083.png,True,False,299,299,L,106.57,0a4cdbfe74430e2bdcbd811667b2185a
Lung_Opacity-2084.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2084.png,Lung_Opacity/images/Lung_Opacity-2084.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2084.png,Lung_Opacity/masks/Lung_Opacity-2084.png,True,False,299,299,L,130.87,5e1546a14481a3c2529c26e575cbbb06
Lung_Opacity-2085.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2085.png,Lung_Opacity/images/Lung_Opacity-2085.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2085.png,Lung_Opacity/masks/Lung_Opacity-2085.png,True,False,299,299,L,116.94,5f8ed1f71083adfcce0b181a272013ee
Lung_Opacity-2086.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2086.png,Lung_Opacity/images/Lung_Opacity-2086.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2086.png,Lung_Opacity/masks/Lung_Opacity-2086.png,True,False,299,299,L,117.76,86dd2f36f0c4e4d9b52d7b5ae298c5e0
Lung_Opacity-2087.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2087.png,Lung_Opacity/images/Lung_Opacity-2087.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2087.png,Lung_Opacity/masks/Lung_Opacity-2087.png,True,False,299,299,L,88.18,076779ebc6bc91c647bccd541c304b87
Lung_Opacity-2088.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2088.png,Lung_Opacity/images/Lung_Opacity-2088.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2088.png,Lung_Opacity/masks/Lung_Opacity-2088.png,True,False,299,299,L,179.11,013909a8b78e8167205d4f0170b7b141
Lung_Opacity-2089.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2089.png,Lung_Opacity/images/Lung_Opacity-2089.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2089.png,Lung_Opacity/masks/Lung_Opacity-2089.png,True,False,299,299,L,127.5,1903d7ef12dcae807999ae30eb67a0e7
Lung_Opacity-209.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-209.png,Lung_Opacity/images/Lung_Opacity-209.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-209.png,Lung_Opacity/masks/Lung_Opacity-209.png,True,False,299,299,L,155.13,22a6c451ef380b3e4e7354691bd0303d
Lung_Opacity-2090.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2090.png,Lung_Opacity/images/Lung_Opacity-2090.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2090.png,Lung_Opacity/masks/Lung_Opacity-2090.png,True,False,299,299,L,123.91,ad36c3b8a094db1e2e4b0a7d70341b7d
Lung_Opacity-2091.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2091.png,Lung_Opacity/images/Lung_Opacity-2091.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2091.png,Lung_Opacity/masks/Lung_Opacity-2091.png,True,False,299,299,L,119.35,960a7c5642d1bedf13254b55665bcfb4
Lung_Opacity-2092.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2092.png,Lung_Opacity/images/Lung_Opacity-2092.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2092.png,Lung_Opacity/masks/Lung_Opacity-2092.png,True,False,299,299,L,163.0,8edfa7c630ef819e970e6887d5e3ab22
Lung_Opacity-2093.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2093.png,Lung_Opacity/images/Lung_Opacity-2093.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2093.png,Lung_Opacity/masks/Lung_Opacity-2093.png,True,False,299,299,L,101.27,b5f8bfb5a1aef7135a86e47cebf437b6
Lung_Opacity-2094.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2094.png,Lung_Opacity/images/Lung_Opacity-2094.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2094.png,Lung_Opacity/masks/Lung_Opacity-2094.png,True,False,299,299,L,107.17,a003513d9c2d20477f6e7722a801fe33
Lung_Opacity-2095.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2095.png,Lung_Opacity/images/Lung_Opacity-2095.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2095.png,Lung_Opacity/masks/Lung_Opacity-2095.png,True,False,299,299,L,134.77,6a5b42a3ea825409e61fa72cc458361c
Lung_Opacity-2096.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2096.png,Lung_Opacity/images/Lung_Opacity-2096.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2096.png,Lung_Opacity/masks/Lung_Opacity-2096.png,True,False,299,299,L,90.7,1ca95c50e936c489e80aa577db7fad2c
Lung_Opacity-2097.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2097.png,Lung_Opacity/images/Lung_Opacity-2097.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2097.png,Lung_Opacity/masks/Lung_Opacity-2097.png,True,False,299,299,L,86.43,17a652d6f6adf2753bf79d7ff76bad90
Lung_Opacity-2098.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2098.png,Lung_Opacity/images/Lung_Opacity-2098.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2098.png,Lung_Opacity/masks/Lung_Opacity-2098.png,True,False,299,299,L,99.48,5748889d4954ae60434634acffe71784
Lung_Opacity-2099.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2099.png,Lung_Opacity/images/Lung_Opacity-2099.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2099.png,Lung_Opacity/masks/Lung_Opacity-2099.png,True,False,299,299,L,158.69,b01d11c519c09cb1ea08387cc41339c6
Lung_Opacity-21.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-21.png,Lung_Opacity/images/Lung_Opacity-21.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-21.png,Lung_Opacity/masks/Lung_Opacity-21.png,True,False,299,299,L,111.74,475e3e00375ef575594bcdfcd0b675f2
Lung_Opacity-210.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-210.png,Lung_Opacity/images/Lung_Opacity-210.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-210.png,Lung_Opacity/masks/Lung_Opacity-210.png,True,False,299,299,L,95.93,677ae0d6ba99edb529db071e069a80c8
Lung_Opacity-2100.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2100.png,Lung_Opacity/images/Lung_Opacity-2100.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2100.png,Lung_Opacity/masks/Lung_Opacity-2100.png,True,False,299,299,L,114.35,bd10db730933e8cebcd9ea5415e11aee
Lung_Opacity-2101.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2101.png,Lung_Opacity/images/Lung_Opacity-2101.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2101.png,Lung_Opacity/masks/Lung_Opacity-2101.png,True,False,299,299,L,121.51,da0f06e2c8d33ffb6780dd34e863c6db
Lung_Opacity-2102.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2102.png,Lung_Opacity/images/Lung_Opacity-2102.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2102.png,Lung_Opacity/masks/Lung_Opacity-2102.png,True,False,299,299,L,131.59,73c23bb1071b80956f2098fc0378ea68
Lung_Opacity-2103.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2103.png,Lung_Opacity/images/Lung_Opacity-2103.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2103.png,Lung_Opacity/masks/Lung_Opacity-2103.png,True,False,299,299,L,118.15,7eb2b870e587296da8831f6c595ddf81
Lung_Opacity-2104.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2104.png,Lung_Opacity/images/Lung_Opacity-2104.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2104.png,Lung_Opacity/masks/Lung_Opacity-2104.png,True,False,299,299,L,120.65,bc000b71fe3c0e005d5a945bbd1e15f0
Lung_Opacity-2105.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2105.png,Lung_Opacity/images/Lung_Opacity-2105.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2105.png,Lung_Opacity/masks/Lung_Opacity-2105.png,True,False,299,299,L,112.76,ab764b03144bb6c155b8384bcfa7e5e2
Lung_Opacity-2106.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2106.png,Lung_Opacity/images/Lung_Opacity-2106.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2106.png,Lung_Opacity/masks/Lung_Opacity-2106.png,True,False,299,299,L,122.79,0f3acaa7901363016756fada873b64f9
Lung_Opacity-2107.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2107.png,Lung_Opacity/images/Lung_Opacity-2107.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2107.png,Lung_Opacity/masks/Lung_Opacity-2107.png,True,False,299,299,L,84.19,9532ca26262c9f32aec08794e862ee81
Lung_Opacity-2108.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2108.png,Lung_Opacity/images/Lung_Opacity-2108.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2108.png,Lung_Opacity/masks/Lung_Opacity-2108.png,True,False,299,299,L,98.33,6c963536e224cefbee067a23f6016dba
Lung_Opacity-2109.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2109.png,Lung_Opacity/images/Lung_Opacity-2109.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2109.png,Lung_Opacity/masks/Lung_Opacity-2109.png,True,False,299,299,L,155.75,e4d56dc68617b2ef03865ac8408f5a32
Lung_Opacity-211.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-211.png,Lung_Opacity/images/Lung_Opacity-211.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-211.png,Lung_Opacity/masks/Lung_Opacity-211.png,True,False,299,299,L,116.86,e171450cbc33200f898db7c7216ca60a
Lung_Opacity-2110.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2110.png,Lung_Opacity/images/Lung_Opacity-2110.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2110.png,Lung_Opacity/masks/Lung_Opacity-2110.png,True,False,299,299,L,114.35,0c70cca62a3a64712e1f4875611c6e64
Lung_Opacity-2111.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2111.png,Lung_Opacity/images/Lung_Opacity-2111.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2111.png,Lung_Opacity/masks/Lung_Opacity-2111.png,True,False,299,299,L,108.11,83542c81296a27a594c661dd961338c0
Lung_Opacity-2112.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2112.png,Lung_Opacity/images/Lung_Opacity-2112.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2112.png,Lung_Opacity/masks/Lung_Opacity-2112.png,True,False,299,299,L,137.92,cab73da22473c1894bcc5680c24cd7de
Lung_Opacity-2113.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2113.png,Lung_Opacity/images/Lung_Opacity-2113.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2113.png,Lung_Opacity/masks/Lung_Opacity-2113.png,True,False,299,299,L,111.69,f1bb1747f3f07ca4150d390ee298aef6
Lung_Opacity-2114.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2114.png,Lung_Opacity/images/Lung_Opacity-2114.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2114.png,Lung_Opacity/masks/Lung_Opacity-2114.png,True,False,299,299,L,147.15,d5ffbab4de7a7f7cccd8f3de16d161b0
Lung_Opacity-2115.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2115.png,Lung_Opacity/images/Lung_Opacity-2115.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2115.png,Lung_Opacity/masks/Lung_Opacity-2115.png,True,False,299,299,L,133.22,14dca2b2189330c72e6e9e0f0d342bdd
Lung_Opacity-2116.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2116.png,Lung_Opacity/images/Lung_Opacity-2116.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2116.png,Lung_Opacity/masks/Lung_Opacity-2116.png,True,False,299,299,L,112.98,6f872acd83db6c6d0b8cdc93d9e2bdd6
Lung_Opacity-2117.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2117.png,Lung_Opacity/images/Lung_Opacity-2117.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2117.png,Lung_Opacity/masks/Lung_Opacity-2117.png,True,False,299,299,L,128.11,dfa5bb9ec3888a96832d03d292dcd98e
Lung_Opacity-2118.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2118.png,Lung_Opacity/images/Lung_Opacity-2118.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2118.png,Lung_Opacity/masks/Lung_Opacity-2118.png,True,False,299,299,L,138.66,10a60d40abbf33320b3a50ce23ce6116
Lung_Opacity-2119.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2119.png,Lung_Opacity/images/Lung_Opacity-2119.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2119.png,Lung_Opacity/masks/Lung_Opacity-2119.png,True,False,299,299,L,115.71,852bd82ff53053e0582561c53b8c954e
Lung_Opacity-212.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-212.png,Lung_Opacity/images/Lung_Opacity-212.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-212.png,Lung_Opacity/masks/Lung_Opacity-212.png,True,False,299,299,L,132.19,2acb3cf2832af84a49fd4da86c2b1da4
Lung_Opacity-2120.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2120.png,Lung_Opacity/images/Lung_Opacity-2120.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2120.png,Lung_Opacity/masks/Lung_Opacity-2120.png,True,False,299,299,L,126.69,f315748bd45e5b8b6a8091cd3c2ec36b
Lung_Opacity-2121.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2121.png,Lung_Opacity/images/Lung_Opacity-2121.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2121.png,Lung_Opacity/masks/Lung_Opacity-2121.png,True,False,299,299,L,120.15,b97c900d3ae228f4913db9e1ed734803
Lung_Opacity-2122.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2122.png,Lung_Opacity/images/Lung_Opacity-2122.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2122.png,Lung_Opacity/masks/Lung_Opacity-2122.png,True,False,299,299,L,111.62,4ad75018eeb4f67717fbd94206e82fce
Lung_Opacity-2123.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2123.png,Lung_Opacity/images/Lung_Opacity-2123.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2123.png,Lung_Opacity/masks/Lung_Opacity-2123.png,True,False,299,299,L,145.01,c88fac98c4bb625eb43c5a60920de649
Lung_Opacity-2124.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2124.png,Lung_Opacity/images/Lung_Opacity-2124.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2124.png,Lung_Opacity/masks/Lung_Opacity-2124.png,True,False,299,299,L,109.74,a0cea4cef66819d2b601536dae06b45a
Lung_Opacity-2125.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2125.png,Lung_Opacity/images/Lung_Opacity-2125.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2125.png,Lung_Opacity/masks/Lung_Opacity-2125.png,True,False,299,299,L,159.51,31102267f655251650b68748e85e03d3
Lung_Opacity-2126.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2126.png,Lung_Opacity/images/Lung_Opacity-2126.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2126.png,Lung_Opacity/masks/Lung_Opacity-2126.png,True,False,299,299,L,172.03,ca50fb2b04ce0e7a6c6c85bc70048a59
Lung_Opacity-2127.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2127.png,Lung_Opacity/images/Lung_Opacity-2127.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2127.png,Lung_Opacity/masks/Lung_Opacity-2127.png,True,False,299,299,L,135.02,83e50af8d12f4e528c53292c507102fb
Lung_Opacity-2128.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2128.png,Lung_Opacity/images/Lung_Opacity-2128.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2128.png,Lung_Opacity/masks/Lung_Opacity-2128.png,True,False,299,299,L,140.27,5f5029fdbc0d4b6d513adde7d874a530
Lung_Opacity-2129.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2129.png,Lung_Opacity/images/Lung_Opacity-2129.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2129.png,Lung_Opacity/masks/Lung_Opacity-2129.png,True,False,299,299,L,144.75,9bd855d56d23e0d11e481e073b28fcff
Lung_Opacity-213.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-213.png,Lung_Opacity/images/Lung_Opacity-213.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-213.png,Lung_Opacity/masks/Lung_Opacity-213.png,True,False,299,299,L,115.7,a172cc40274099e561619891270e6f8a
Lung_Opacity-2130.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2130.png,Lung_Opacity/images/Lung_Opacity-2130.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2130.png,Lung_Opacity/masks/Lung_Opacity-2130.png,True,False,299,299,L,113.52,7ee0365eeca27390f40e63c25683f737
Lung_Opacity-2131.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2131.png,Lung_Opacity/images/Lung_Opacity-2131.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2131.png,Lung_Opacity/masks/Lung_Opacity-2131.png,True,False,299,299,L,121.29,e34728291cac0f965901889fda239ffe
Lung_Opacity-2132.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2132.png,Lung_Opacity/images/Lung_Opacity-2132.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2132.png,Lung_Opacity/masks/Lung_Opacity-2132.png,True,False,299,299,L,107.11,ec526b13afecfa5665dca8e2c627879c
Lung_Opacity-2133.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2133.png,Lung_Opacity/images/Lung_Opacity-2133.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2133.png,Lung_Opacity/masks/Lung_Opacity-2133.png,True,False,299,299,L,111.24,25c47579564a77507fe80a9bf2b2aa74
Lung_Opacity-2134.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2134.png,Lung_Opacity/images/Lung_Opacity-2134.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2134.png,Lung_Opacity/masks/Lung_Opacity-2134.png,True,False,299,299,L,78.39,5cc93876f9d89c03b22e224d442864a9
Lung_Opacity-2135.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2135.png,Lung_Opacity/images/Lung_Opacity-2135.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2135.png,Lung_Opacity/masks/Lung_Opacity-2135.png,True,False,299,299,L,151.49,b873b5edfcf6cc3b4ca14985456ef689
Lung_Opacity-2136.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2136.png,Lung_Opacity/images/Lung_Opacity-2136.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2136.png,Lung_Opacity/masks/Lung_Opacity-2136.png,True,False,299,299,L,153.75,f8f8980747a313cb70cd5ca54c17b371
Lung_Opacity-2137.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2137.png,Lung_Opacity/images/Lung_Opacity-2137.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2137.png,Lung_Opacity/masks/Lung_Opacity-2137.png,True,False,299,299,L,109.62,a2d1ab6200ed8448511034d5e4c7d64a
Lung_Opacity-2138.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2138.png,Lung_Opacity/images/Lung_Opacity-2138.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2138.png,Lung_Opacity/masks/Lung_Opacity-2138.png,True,False,299,299,L,106.68,99e73a6505e589f753b3cd1d43173786
Lung_Opacity-2139.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2139.png,Lung_Opacity/images/Lung_Opacity-2139.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2139.png,Lung_Opacity/masks/Lung_Opacity-2139.png,True,False,299,299,L,106.91,f947f185b0c5d6eee4870569e56bf59f
Lung_Opacity-214.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-214.png,Lung_Opacity/images/Lung_Opacity-214.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-214.png,Lung_Opacity/masks/Lung_Opacity-214.png,True,False,299,299,L,118.08,eba1d83cdad320a650f05711878882df
Lung_Opacity-2140.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2140.png,Lung_Opacity/images/Lung_Opacity-2140.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2140.png,Lung_Opacity/masks/Lung_Opacity-2140.png,True,False,299,299,L,162.88,55e5118ea9ca1eca7b64df7d98d19e8b
Lung_Opacity-2141.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2141.png,Lung_Opacity/images/Lung_Opacity-2141.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2141.png,Lung_Opacity/masks/Lung_Opacity-2141.png,True,False,299,299,L,159.74,9e037280fc22c0ca68db464c813f55e8
Lung_Opacity-2142.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2142.png,Lung_Opacity/images/Lung_Opacity-2142.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2142.png,Lung_Opacity/masks/Lung_Opacity-2142.png,True,False,299,299,L,161.83,f86040fa069e829af868730dd6c683e8
Lung_Opacity-2143.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2143.png,Lung_Opacity/images/Lung_Opacity-2143.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2143.png,Lung_Opacity/masks/Lung_Opacity-2143.png,True,False,299,299,L,156.44,bc5fc16b3ac55d4e919a5b9032514967
Lung_Opacity-2144.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2144.png,Lung_Opacity/images/Lung_Opacity-2144.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2144.png,Lung_Opacity/masks/Lung_Opacity-2144.png,True,False,299,299,L,88.36,d2ba37e464fd2adf4b5ebd5b64f3b4b6
Lung_Opacity-2145.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2145.png,Lung_Opacity/images/Lung_Opacity-2145.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2145.png,Lung_Opacity/masks/Lung_Opacity-2145.png,True,False,299,299,L,149.52,715944efff125ab2f27324f0c48afa4c
Lung_Opacity-2146.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2146.png,Lung_Opacity/images/Lung_Opacity-2146.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2146.png,Lung_Opacity/masks/Lung_Opacity-2146.png,True,False,299,299,L,136.83,43d710f7733997f7c856f499c1f12487
Lung_Opacity-2147.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2147.png,Lung_Opacity/images/Lung_Opacity-2147.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2147.png,Lung_Opacity/masks/Lung_Opacity-2147.png,True,False,299,299,L,161.02,85270328ad6606066963ed008d8545e5
Lung_Opacity-2148.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2148.png,Lung_Opacity/images/Lung_Opacity-2148.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2148.png,Lung_Opacity/masks/Lung_Opacity-2148.png,True,False,299,299,L,136.25,dc18804ef6a6ad7b01968613253b91e6
Lung_Opacity-2149.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2149.png,Lung_Opacity/images/Lung_Opacity-2149.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2149.png,Lung_Opacity/masks/Lung_Opacity-2149.png,True,False,299,299,L,106.48,1b16b3bb2ec12cdc913bfb770685aeea
Lung_Opacity-215.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-215.png,Lung_Opacity/images/Lung_Opacity-215.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-215.png,Lung_Opacity/masks/Lung_Opacity-215.png,True,False,299,299,L,102.2,7cb29030de2737772c45e3104f9eafa4
Lung_Opacity-2150.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2150.png,Lung_Opacity/images/Lung_Opacity-2150.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2150.png,Lung_Opacity/masks/Lung_Opacity-2150.png,True,False,299,299,L,148.12,63f8faeb00149132923b5e105613f436
Lung_Opacity-2151.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2151.png,Lung_Opacity/images/Lung_Opacity-2151.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2151.png,Lung_Opacity/masks/Lung_Opacity-2151.png,True,False,299,299,L,128.41,bfbaf2bf1f48857f5572b55fc68c2a54
Lung_Opacity-2152.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2152.png,Lung_Opacity/images/Lung_Opacity-2152.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2152.png,Lung_Opacity/masks/Lung_Opacity-2152.png,True,False,299,299,L,131.05,38496687ff29388adfc5021a454855fb
Lung_Opacity-2153.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2153.png,Lung_Opacity/images/Lung_Opacity-2153.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2153.png,Lung_Opacity/masks/Lung_Opacity-2153.png,True,False,299,299,L,126.68,340cee71062750581f8e32841839ce45
Lung_Opacity-2154.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2154.png,Lung_Opacity/images/Lung_Opacity-2154.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2154.png,Lung_Opacity/masks/Lung_Opacity-2154.png,True,False,299,299,L,120.3,b14b4aa31534757275e06807352cd6c7
Lung_Opacity-2155.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2155.png,Lung_Opacity/images/Lung_Opacity-2155.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2155.png,Lung_Opacity/masks/Lung_Opacity-2155.png,True,False,299,299,L,104.98,2ee9742a9aa06d2610d4eb55af879df1
Lung_Opacity-2156.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2156.png,Lung_Opacity/images/Lung_Opacity-2156.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2156.png,Lung_Opacity/masks/Lung_Opacity-2156.png,True,False,299,299,L,129.71,fd35293f71b94825e8f7a5b691a295f2
Lung_Opacity-2157.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2157.png,Lung_Opacity/images/Lung_Opacity-2157.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2157.png,Lung_Opacity/masks/Lung_Opacity-2157.png,True,False,299,299,L,105.04,bb32dfb8a59b0d9d76ee80bab82f6782
Lung_Opacity-2158.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2158.png,Lung_Opacity/images/Lung_Opacity-2158.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2158.png,Lung_Opacity/masks/Lung_Opacity-2158.png,True,False,299,299,L,151.49,92e9b440189cc23d31198ffd5b438bba
Lung_Opacity-2159.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2159.png,Lung_Opacity/images/Lung_Opacity-2159.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2159.png,Lung_Opacity/masks/Lung_Opacity-2159.png,True,False,299,299,L,115.6,edc80637ac44518d7eaa053517781e0d
Lung_Opacity-216.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-216.png,Lung_Opacity/images/Lung_Opacity-216.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-216.png,Lung_Opacity/masks/Lung_Opacity-216.png,True,False,299,299,L,114.28,5551e48dc109b095ee5e57a2f3a62d0e
Lung_Opacity-2160.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2160.png,Lung_Opacity/images/Lung_Opacity-2160.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2160.png,Lung_Opacity/masks/Lung_Opacity-2160.png,True,False,299,299,L,108.82,f1b60f1f3b32fe2d60981d13deba97a9
Lung_Opacity-2161.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2161.png,Lung_Opacity/images/Lung_Opacity-2161.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2161.png,Lung_Opacity/masks/Lung_Opacity-2161.png,True,False,299,299,L,124.65,e4d9b9190fde67b15185a8bd23a2efad
Lung_Opacity-2162.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2162.png,Lung_Opacity/images/Lung_Opacity-2162.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2162.png,Lung_Opacity/masks/Lung_Opacity-2162.png,True,False,299,299,L,146.5,74878223b0100ff08bd35cd61e02a246
Lung_Opacity-2163.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2163.png,Lung_Opacity/images/Lung_Opacity-2163.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2163.png,Lung_Opacity/masks/Lung_Opacity-2163.png,True,False,299,299,L,106.62,cd62eb72eead12d7bb9107f15d602e7f
Lung_Opacity-2164.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2164.png,Lung_Opacity/images/Lung_Opacity-2164.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2164.png,Lung_Opacity/masks/Lung_Opacity-2164.png,True,False,299,299,L,103.58,a1e11dcffa7c8ac106fbf9d173e34a89
Lung_Opacity-2165.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2165.png,Lung_Opacity/images/Lung_Opacity-2165.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2165.png,Lung_Opacity/masks/Lung_Opacity-2165.png,True,False,299,299,L,120.12,c8e6795799911127aa3f69bf9bd935c3
Lung_Opacity-2166.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2166.png,Lung_Opacity/images/Lung_Opacity-2166.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2166.png,Lung_Opacity/masks/Lung_Opacity-2166.png,True,False,299,299,L,93.86,272c9d9f674c64bb6ec9ce04f3c316e1
Lung_Opacity-2167.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2167.png,Lung_Opacity/images/Lung_Opacity-2167.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2167.png,Lung_Opacity/masks/Lung_Opacity-2167.png,True,False,299,299,L,118.16,79cfedd96c0436a0a0e95da3f34a023b
Lung_Opacity-2168.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2168.png,Lung_Opacity/images/Lung_Opacity-2168.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2168.png,Lung_Opacity/masks/Lung_Opacity-2168.png,True,False,299,299,L,92.73,3de500dc40e7e7217c93f4117164dacd
Lung_Opacity-2169.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2169.png,Lung_Opacity/images/Lung_Opacity-2169.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2169.png,Lung_Opacity/masks/Lung_Opacity-2169.png,True,False,299,299,L,133.83,8855158fd73890b23c275439d38e22ae
Lung_Opacity-217.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-217.png,Lung_Opacity/images/Lung_Opacity-217.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-217.png,Lung_Opacity/masks/Lung_Opacity-217.png,True,False,299,299,L,95.78,285fb2faab61ab90f9f8eda474afb2d5
Lung_Opacity-2170.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2170.png,Lung_Opacity/images/Lung_Opacity-2170.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2170.png,Lung_Opacity/masks/Lung_Opacity-2170.png,True,False,299,299,L,130.28,990dc79814126e76eb320088504dada5
Lung_Opacity-2171.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2171.png,Lung_Opacity/images/Lung_Opacity-2171.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2171.png,Lung_Opacity/masks/Lung_Opacity-2171.png,True,False,299,299,L,119.68,e500a1d710a7b318c1e01d8ec18d4e8d
Lung_Opacity-2172.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2172.png,Lung_Opacity/images/Lung_Opacity-2172.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2172.png,Lung_Opacity/masks/Lung_Opacity-2172.png,True,False,299,299,L,112.08,d0e8a12e10d2672fa7dfa378e1ad9afa
Lung_Opacity-2173.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2173.png,Lung_Opacity/images/Lung_Opacity-2173.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2173.png,Lung_Opacity/masks/Lung_Opacity-2173.png,True,False,299,299,L,116.12,8d66968efd695faae540f65caeb96e6e
Lung_Opacity-2174.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2174.png,Lung_Opacity/images/Lung_Opacity-2174.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2174.png,Lung_Opacity/masks/Lung_Opacity-2174.png,True,False,299,299,L,144.15,5dd173bf3883d7bdbe200d940d3741a3
Lung_Opacity-2175.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2175.png,Lung_Opacity/images/Lung_Opacity-2175.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2175.png,Lung_Opacity/masks/Lung_Opacity-2175.png,True,False,299,299,L,139.77,ad743a0ac9a79b100fea0e7ba5bcf444
Lung_Opacity-2176.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2176.png,Lung_Opacity/images/Lung_Opacity-2176.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2176.png,Lung_Opacity/masks/Lung_Opacity-2176.png,True,False,299,299,L,119.77,763a5dc04878a1f38a0a9b9894b7bbaa
Lung_Opacity-2177.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2177.png,Lung_Opacity/images/Lung_Opacity-2177.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2177.png,Lung_Opacity/masks/Lung_Opacity-2177.png,True,False,299,299,L,146.2,9878d250ad75358e62aa3efbd00ae8e9
Lung_Opacity-2178.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2178.png,Lung_Opacity/images/Lung_Opacity-2178.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2178.png,Lung_Opacity/masks/Lung_Opacity-2178.png,True,False,299,299,L,120.46,533e8d2bb1c93af8d195c8f36ffc9bf3
Lung_Opacity-2179.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2179.png,Lung_Opacity/images/Lung_Opacity-2179.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2179.png,Lung_Opacity/masks/Lung_Opacity-2179.png,True,False,299,299,L,133.15,db08f6ceafd21490b7c5082e4aa792ec
Lung_Opacity-218.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-218.png,Lung_Opacity/images/Lung_Opacity-218.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-218.png,Lung_Opacity/masks/Lung_Opacity-218.png,True,False,299,299,L,164.59,8b13369413d03cfc5e3019e2bc73b5ae
Lung_Opacity-2180.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2180.png,Lung_Opacity/images/Lung_Opacity-2180.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2180.png,Lung_Opacity/masks/Lung_Opacity-2180.png,True,False,299,299,L,104.07,7dda8023fbd757dfe170fc210cd56398
Lung_Opacity-2181.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2181.png,Lung_Opacity/images/Lung_Opacity-2181.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2181.png,Lung_Opacity/masks/Lung_Opacity-2181.png,True,False,299,299,L,136.18,7dfbfa1091b36497283366e48f429f64
Lung_Opacity-2182.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2182.png,Lung_Opacity/images/Lung_Opacity-2182.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2182.png,Lung_Opacity/masks/Lung_Opacity-2182.png,True,False,299,299,L,108.58,d2925cd7d24728c7b1968ccac74454d5
Lung_Opacity-2183.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2183.png,Lung_Opacity/images/Lung_Opacity-2183.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2183.png,Lung_Opacity/masks/Lung_Opacity-2183.png,True,False,299,299,L,110.61,80217732b9b378bc653214b815e04313
Lung_Opacity-2184.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2184.png,Lung_Opacity/images/Lung_Opacity-2184.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2184.png,Lung_Opacity/masks/Lung_Opacity-2184.png,True,False,299,299,L,145.36,210fc6577c7148192f5f65f2e88d2afc
Lung_Opacity-2185.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2185.png,Lung_Opacity/images/Lung_Opacity-2185.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2185.png,Lung_Opacity/masks/Lung_Opacity-2185.png,True,False,299,299,L,144.95,9b898a07735eec8ec98a383710a4e572
Lung_Opacity-2186.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2186.png,Lung_Opacity/images/Lung_Opacity-2186.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2186.png,Lung_Opacity/masks/Lung_Opacity-2186.png,True,False,299,299,L,103.32,446e5b4f817bbda9c35677b800780e54
Lung_Opacity-2187.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2187.png,Lung_Opacity/images/Lung_Opacity-2187.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2187.png,Lung_Opacity/masks/Lung_Opacity-2187.png,True,False,299,299,L,116.59,4a067175f445d67eb43cd4f071eb1a8e
Lung_Opacity-2188.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2188.png,Lung_Opacity/images/Lung_Opacity-2188.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2188.png,Lung_Opacity/masks/Lung_Opacity-2188.png,True,False,299,299,L,120.86,0a5fdbf309fc62001bb6c4d2dc9e61ab
Lung_Opacity-2189.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2189.png,Lung_Opacity/images/Lung_Opacity-2189.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2189.png,Lung_Opacity/masks/Lung_Opacity-2189.png,True,False,299,299,L,127.81,72b4457d89f77514b7bf40afe2e29bc0
Lung_Opacity-219.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-219.png,Lung_Opacity/images/Lung_Opacity-219.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-219.png,Lung_Opacity/masks/Lung_Opacity-219.png,True,False,299,299,L,103.99,3d30572e6bebc5694e33d8d64158346e
Lung_Opacity-2190.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2190.png,Lung_Opacity/images/Lung_Opacity-2190.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2190.png,Lung_Opacity/masks/Lung_Opacity-2190.png,True,False,299,299,L,149.54,7bfc1722f8a9b8052d71ab1653770dc4
Lung_Opacity-2191.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2191.png,Lung_Opacity/images/Lung_Opacity-2191.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2191.png,Lung_Opacity/masks/Lung_Opacity-2191.png,True,False,299,299,L,112.08,6a73bf0178872d3e41ec59dfbd398b97
Lung_Opacity-2192.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2192.png,Lung_Opacity/images/Lung_Opacity-2192.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2192.png,Lung_Opacity/masks/Lung_Opacity-2192.png,True,False,299,299,L,134.38,cce31429d6d74357d78900574977e04a
Lung_Opacity-2193.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2193.png,Lung_Opacity/images/Lung_Opacity-2193.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2193.png,Lung_Opacity/masks/Lung_Opacity-2193.png,True,False,299,299,L,107.66,d0f04078cd79601ba6cc3ec82ff6f129
Lung_Opacity-2194.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2194.png,Lung_Opacity/images/Lung_Opacity-2194.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2194.png,Lung_Opacity/masks/Lung_Opacity-2194.png,True,False,299,299,L,133.74,81a280438d5c3a64f7db6cba77098b01
Lung_Opacity-2195.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2195.png,Lung_Opacity/images/Lung_Opacity-2195.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2195.png,Lung_Opacity/masks/Lung_Opacity-2195.png,True,False,299,299,L,162.68,6c92486825784158c517c10f8200ef5c
Lung_Opacity-2196.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2196.png,Lung_Opacity/images/Lung_Opacity-2196.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2196.png,Lung_Opacity/masks/Lung_Opacity-2196.png,True,False,299,299,L,131.32,c7887d3e91b1a7a1b298da187524bb5a
Lung_Opacity-2197.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2197.png,Lung_Opacity/images/Lung_Opacity-2197.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2197.png,Lung_Opacity/masks/Lung_Opacity-2197.png,True,False,299,299,L,118.55,e4eed336acc80222055ffdd63152349c
Lung_Opacity-2198.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2198.png,Lung_Opacity/images/Lung_Opacity-2198.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2198.png,Lung_Opacity/masks/Lung_Opacity-2198.png,True,False,299,299,L,128.47,5b1ca269a3760dd13422229a42588d39
Lung_Opacity-2199.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2199.png,Lung_Opacity/images/Lung_Opacity-2199.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2199.png,Lung_Opacity/masks/Lung_Opacity-2199.png,True,False,299,299,L,133.36,7094624b8926d92ad46b54dde2f46419
Lung_Opacity-22.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-22.png,Lung_Opacity/images/Lung_Opacity-22.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-22.png,Lung_Opacity/masks/Lung_Opacity-22.png,True,False,299,299,L,112.22,dea8e513d2e337992bddc39cc963aa02
Lung_Opacity-220.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-220.png,Lung_Opacity/images/Lung_Opacity-220.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-220.png,Lung_Opacity/masks/Lung_Opacity-220.png,True,False,299,299,L,140.11,2470f3c188576a7c252758410ee954b5
Lung_Opacity-2200.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2200.png,Lung_Opacity/images/Lung_Opacity-2200.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2200.png,Lung_Opacity/masks/Lung_Opacity-2200.png,True,False,299,299,L,61.23,969748718bd394e1b3b77aafa023bdf2
Lung_Opacity-2201.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2201.png,Lung_Opacity/images/Lung_Opacity-2201.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2201.png,Lung_Opacity/masks/Lung_Opacity-2201.png,True,False,299,299,L,152.66,41680e29a31582ee5fd4a963d26485e6
Lung_Opacity-2202.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2202.png,Lung_Opacity/images/Lung_Opacity-2202.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2202.png,Lung_Opacity/masks/Lung_Opacity-2202.png,True,False,299,299,L,119.11,6a705d32060914068cb4ae06f8950443
Lung_Opacity-2203.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2203.png,Lung_Opacity/images/Lung_Opacity-2203.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2203.png,Lung_Opacity/masks/Lung_Opacity-2203.png,True,False,299,299,L,147.23,d199bb14421aea5548849e5091c6bd1b
Lung_Opacity-2204.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2204.png,Lung_Opacity/images/Lung_Opacity-2204.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2204.png,Lung_Opacity/masks/Lung_Opacity-2204.png,True,False,299,299,L,100.46,8451e1d82e65f1f6d0068bce19953315
Lung_Opacity-2205.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2205.png,Lung_Opacity/images/Lung_Opacity-2205.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2205.png,Lung_Opacity/masks/Lung_Opacity-2205.png,True,False,299,299,L,107.13,87a4e8f39d8ad1916ce52be8dda63204
Lung_Opacity-2206.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2206.png,Lung_Opacity/images/Lung_Opacity-2206.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2206.png,Lung_Opacity/masks/Lung_Opacity-2206.png,True,False,299,299,L,111.82,a425903efdb03207ce985ca1db4f9148
Lung_Opacity-2207.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2207.png,Lung_Opacity/images/Lung_Opacity-2207.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2207.png,Lung_Opacity/masks/Lung_Opacity-2207.png,True,False,299,299,L,136.13,672476f55a8815db3c9a623dce961d9e
Lung_Opacity-2208.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2208.png,Lung_Opacity/images/Lung_Opacity-2208.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2208.png,Lung_Opacity/masks/Lung_Opacity-2208.png,True,False,299,299,L,163.25,3b741bbb9c3bc5d064d541ba47ef9031
Lung_Opacity-2209.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2209.png,Lung_Opacity/images/Lung_Opacity-2209.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2209.png,Lung_Opacity/masks/Lung_Opacity-2209.png,True,False,299,299,L,140.15,a09573704400adcab7eb77fdf5e7c5a7
Lung_Opacity-221.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-221.png,Lung_Opacity/images/Lung_Opacity-221.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-221.png,Lung_Opacity/masks/Lung_Opacity-221.png,True,False,299,299,L,128.95,027e427982b7d6dc1c9a261f73147c4c
Lung_Opacity-2210.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2210.png,Lung_Opacity/images/Lung_Opacity-2210.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2210.png,Lung_Opacity/masks/Lung_Opacity-2210.png,True,False,299,299,L,119.18,dd643f833466baa315fe2c46782e0d70
Lung_Opacity-2211.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2211.png,Lung_Opacity/images/Lung_Opacity-2211.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2211.png,Lung_Opacity/masks/Lung_Opacity-2211.png,True,False,299,299,L,150.46,f8ecbac505b2a8ad750a9fd52bfc992b
Lung_Opacity-2212.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2212.png,Lung_Opacity/images/Lung_Opacity-2212.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2212.png,Lung_Opacity/masks/Lung_Opacity-2212.png,True,False,299,299,L,101.39,6178dc9425c8e87163c5e1e257fed7e8
Lung_Opacity-2213.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2213.png,Lung_Opacity/images/Lung_Opacity-2213.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2213.png,Lung_Opacity/masks/Lung_Opacity-2213.png,True,False,299,299,L,107.36,eefbdfa1c0445727de1c2cc068e76823
Lung_Opacity-2214.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2214.png,Lung_Opacity/images/Lung_Opacity-2214.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2214.png,Lung_Opacity/masks/Lung_Opacity-2214.png,True,False,299,299,L,107.33,e82a18b84ca2fa272f89aab78636c1e9
Lung_Opacity-2215.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2215.png,Lung_Opacity/images/Lung_Opacity-2215.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2215.png,Lung_Opacity/masks/Lung_Opacity-2215.png,True,False,299,299,L,133.98,e744fd5d7cabb2f9bc7a0b738f80baa0
Lung_Opacity-2216.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2216.png,Lung_Opacity/images/Lung_Opacity-2216.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2216.png,Lung_Opacity/masks/Lung_Opacity-2216.png,True,False,299,299,L,116.74,cbaae867c8c58daefb999b04381af79d
Lung_Opacity-2217.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2217.png,Lung_Opacity/images/Lung_Opacity-2217.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2217.png,Lung_Opacity/masks/Lung_Opacity-2217.png,True,False,299,299,L,102.69,1abc502bd4ff6cb6529c2bbfab3354e7
Lung_Opacity-2218.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2218.png,Lung_Opacity/images/Lung_Opacity-2218.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2218.png,Lung_Opacity/masks/Lung_Opacity-2218.png,True,False,299,299,L,114.79,f753c2fa91535c02d23c7086c587c7f5
Lung_Opacity-2219.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2219.png,Lung_Opacity/images/Lung_Opacity-2219.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2219.png,Lung_Opacity/masks/Lung_Opacity-2219.png,True,False,299,299,L,165.83,6b69f2c3adee53119e1f6ff35b187cec
Lung_Opacity-222.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-222.png,Lung_Opacity/images/Lung_Opacity-222.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-222.png,Lung_Opacity/masks/Lung_Opacity-222.png,True,False,299,299,L,137.4,202cfd01fbcfe5b447a383bd2d815b46
Lung_Opacity-2220.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2220.png,Lung_Opacity/images/Lung_Opacity-2220.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2220.png,Lung_Opacity/masks/Lung_Opacity-2220.png,True,False,299,299,L,154.1,546a081ea616469394a243988cc33722
Lung_Opacity-2221.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2221.png,Lung_Opacity/images/Lung_Opacity-2221.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2221.png,Lung_Opacity/masks/Lung_Opacity-2221.png,True,False,299,299,L,123.6,91af50cb86b5a6bcee3621b35e5e51f5
Lung_Opacity-2222.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2222.png,Lung_Opacity/images/Lung_Opacity-2222.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2222.png,Lung_Opacity/masks/Lung_Opacity-2222.png,True,False,299,299,L,103.89,1b47eecd7caf692b598f84d4e379df03
Lung_Opacity-2223.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2223.png,Lung_Opacity/images/Lung_Opacity-2223.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2223.png,Lung_Opacity/masks/Lung_Opacity-2223.png,True,False,299,299,L,109.89,c9d98cc7f2d0c14dabfb0383683c6611
Lung_Opacity-2224.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2224.png,Lung_Opacity/images/Lung_Opacity-2224.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2224.png,Lung_Opacity/masks/Lung_Opacity-2224.png,True,False,299,299,L,97.07,49b0ddf1754061137561c05288803e84
Lung_Opacity-2225.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2225.png,Lung_Opacity/images/Lung_Opacity-2225.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2225.png,Lung_Opacity/masks/Lung_Opacity-2225.png,True,False,299,299,L,108.64,4f5ecda1eb2825541270945ad006beec
Lung_Opacity-2226.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2226.png,Lung_Opacity/images/Lung_Opacity-2226.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2226.png,Lung_Opacity/masks/Lung_Opacity-2226.png,True,False,299,299,L,87.11,219b04bef51d4dd36e01cefffd85d914
Lung_Opacity-2227.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2227.png,Lung_Opacity/images/Lung_Opacity-2227.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2227.png,Lung_Opacity/masks/Lung_Opacity-2227.png,True,False,299,299,L,135.79,b84e89c40dfc0d6fbb7aa0fc3cf488d9
Lung_Opacity-2228.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2228.png,Lung_Opacity/images/Lung_Opacity-2228.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2228.png,Lung_Opacity/masks/Lung_Opacity-2228.png,True,False,299,299,L,155.47,b56df2faf8663ad534ba6a47da03b42f
Lung_Opacity-2229.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2229.png,Lung_Opacity/images/Lung_Opacity-2229.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2229.png,Lung_Opacity/masks/Lung_Opacity-2229.png,True,False,299,299,L,107.3,1f84e2bb6d11a8f3c4ac5ea3ef4b1fcc
Lung_Opacity-223.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-223.png,Lung_Opacity/images/Lung_Opacity-223.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-223.png,Lung_Opacity/masks/Lung_Opacity-223.png,True,False,299,299,L,100.03,3df23a308274cd8fea04aa80ee48a775
Lung_Opacity-2230.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2230.png,Lung_Opacity/images/Lung_Opacity-2230.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2230.png,Lung_Opacity/masks/Lung_Opacity-2230.png,True,False,299,299,L,115.51,93a8d9bca6503497d70f7323075d8481
Lung_Opacity-2231.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2231.png,Lung_Opacity/images/Lung_Opacity-2231.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2231.png,Lung_Opacity/masks/Lung_Opacity-2231.png,True,False,299,299,L,104.92,c974808f75e0a99b5d2e17daab5ff3cf
Lung_Opacity-2232.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2232.png,Lung_Opacity/images/Lung_Opacity-2232.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2232.png,Lung_Opacity/masks/Lung_Opacity-2232.png,True,False,299,299,L,128.9,e9af2545ada3b70e3f83d56233e2bcf3
Lung_Opacity-2233.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2233.png,Lung_Opacity/images/Lung_Opacity-2233.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2233.png,Lung_Opacity/masks/Lung_Opacity-2233.png,True,False,299,299,L,129.48,295591ada3d8436b3e2b844a08370079
Lung_Opacity-2234.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2234.png,Lung_Opacity/images/Lung_Opacity-2234.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2234.png,Lung_Opacity/masks/Lung_Opacity-2234.png,True,False,299,299,L,163.15,b3a9d09884ce6fee8ceae8698fcba939
Lung_Opacity-2235.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2235.png,Lung_Opacity/images/Lung_Opacity-2235.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2235.png,Lung_Opacity/masks/Lung_Opacity-2235.png,True,False,299,299,L,95.82,9e52b2ebd13decd950177b3aab0cb067
Lung_Opacity-2236.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2236.png,Lung_Opacity/images/Lung_Opacity-2236.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2236.png,Lung_Opacity/masks/Lung_Opacity-2236.png,True,False,299,299,L,108.47,b0ae77e82cda3611b13b63a8c77f53d4
Lung_Opacity-2237.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2237.png,Lung_Opacity/images/Lung_Opacity-2237.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2237.png,Lung_Opacity/masks/Lung_Opacity-2237.png,True,False,299,299,L,133.36,86217b8880ca00fbb86b5ed8cb5f2636
Lung_Opacity-2238.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2238.png,Lung_Opacity/images/Lung_Opacity-2238.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2238.png,Lung_Opacity/masks/Lung_Opacity-2238.png,True,False,299,299,L,108.89,0f43425896925d7ec80d4840de5f86be
Lung_Opacity-2239.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2239.png,Lung_Opacity/images/Lung_Opacity-2239.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2239.png,Lung_Opacity/masks/Lung_Opacity-2239.png,True,False,299,299,L,94.25,8e093e2abe042cc3f837573b4ba4edfb
Lung_Opacity-224.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-224.png,Lung_Opacity/images/Lung_Opacity-224.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-224.png,Lung_Opacity/masks/Lung_Opacity-224.png,True,False,299,299,L,117.19,b4792cda6eb19a18c312100ec92e25df
Lung_Opacity-2240.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2240.png,Lung_Opacity/images/Lung_Opacity-2240.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2240.png,Lung_Opacity/masks/Lung_Opacity-2240.png,True,False,299,299,L,135.66,44994d41d4225abcdcdb6c135c010c7d
Lung_Opacity-2241.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2241.png,Lung_Opacity/images/Lung_Opacity-2241.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2241.png,Lung_Opacity/masks/Lung_Opacity-2241.png,True,False,299,299,L,107.2,61a8317a3150a90c0403017bd10742a2
Lung_Opacity-2242.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2242.png,Lung_Opacity/images/Lung_Opacity-2242.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2242.png,Lung_Opacity/masks/Lung_Opacity-2242.png,True,False,299,299,L,117.36,9eca086907b3c8c0f068dc70d195144d
Lung_Opacity-2243.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2243.png,Lung_Opacity/images/Lung_Opacity-2243.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2243.png,Lung_Opacity/masks/Lung_Opacity-2243.png,True,False,299,299,L,108.98,93c638cff812a1575455a9e41e99885c
Lung_Opacity-2244.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2244.png,Lung_Opacity/images/Lung_Opacity-2244.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2244.png,Lung_Opacity/masks/Lung_Opacity-2244.png,True,False,299,299,L,122.36,932fb7acd85150c722d44937c9e11100
Lung_Opacity-2245.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2245.png,Lung_Opacity/images/Lung_Opacity-2245.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2245.png,Lung_Opacity/masks/Lung_Opacity-2245.png,True,False,299,299,L,121.09,6c474e30d1b9eacc7b7954d527b9d070
Lung_Opacity-2246.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2246.png,Lung_Opacity/images/Lung_Opacity-2246.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2246.png,Lung_Opacity/masks/Lung_Opacity-2246.png,True,False,299,299,L,140.93,7087ffca7396133c7ddee42449cc4a7e
Lung_Opacity-2247.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2247.png,Lung_Opacity/images/Lung_Opacity-2247.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2247.png,Lung_Opacity/masks/Lung_Opacity-2247.png,True,False,299,299,L,126.87,4d161d5ee43434e9d205257570291612
Lung_Opacity-2248.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2248.png,Lung_Opacity/images/Lung_Opacity-2248.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2248.png,Lung_Opacity/masks/Lung_Opacity-2248.png,True,False,299,299,L,104.33,aacf28717b6afe9779b1bd80e7b44c8e
Lung_Opacity-2249.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2249.png,Lung_Opacity/images/Lung_Opacity-2249.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2249.png,Lung_Opacity/masks/Lung_Opacity-2249.png,True,False,299,299,L,169.91,006f8f5cbb6f886698fee344f268e641
Lung_Opacity-225.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-225.png,Lung_Opacity/images/Lung_Opacity-225.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-225.png,Lung_Opacity/masks/Lung_Opacity-225.png,True,False,299,299,L,110.57,4c5184e51bbc9aeb3539981b37d7885f
Lung_Opacity-2250.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2250.png,Lung_Opacity/images/Lung_Opacity-2250.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2250.png,Lung_Opacity/masks/Lung_Opacity-2250.png,True,False,299,299,L,160.14,c6f93a6a4f7e41f21812e90edb8af6be
Lung_Opacity-2251.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2251.png,Lung_Opacity/images/Lung_Opacity-2251.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2251.png,Lung_Opacity/masks/Lung_Opacity-2251.png,True,False,299,299,L,119.22,498ee91c3cbe43383d8191242c9f05c1
Lung_Opacity-2252.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2252.png,Lung_Opacity/images/Lung_Opacity-2252.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2252.png,Lung_Opacity/masks/Lung_Opacity-2252.png,True,False,299,299,L,119.33,e86b23b5f7fa5658cb022648da36494b
Lung_Opacity-2253.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2253.png,Lung_Opacity/images/Lung_Opacity-2253.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2253.png,Lung_Opacity/masks/Lung_Opacity-2253.png,True,False,299,299,L,109.68,e10083933ca79a06d323ed7f2e6d8be9
Lung_Opacity-2254.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2254.png,Lung_Opacity/images/Lung_Opacity-2254.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2254.png,Lung_Opacity/masks/Lung_Opacity-2254.png,True,False,299,299,L,182.56,27a1e4b49b17185dbe9d5bd22e4d8fe8
Lung_Opacity-2255.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2255.png,Lung_Opacity/images/Lung_Opacity-2255.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2255.png,Lung_Opacity/masks/Lung_Opacity-2255.png,True,False,299,299,L,107.76,475e63c70475487b179a397f000b5cd6
Lung_Opacity-2256.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2256.png,Lung_Opacity/images/Lung_Opacity-2256.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2256.png,Lung_Opacity/masks/Lung_Opacity-2256.png,True,False,299,299,L,116.32,a403a0b28aaa627ffd21bb2152e03d88
Lung_Opacity-2257.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2257.png,Lung_Opacity/images/Lung_Opacity-2257.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2257.png,Lung_Opacity/masks/Lung_Opacity-2257.png,True,False,299,299,L,113.75,2de4ef99c5730a584d2ed95cb796c23d
Lung_Opacity-2258.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2258.png,Lung_Opacity/images/Lung_Opacity-2258.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2258.png,Lung_Opacity/masks/Lung_Opacity-2258.png,True,False,299,299,L,130.54,b5e3ccd67a7c4cec4396007485354472
Lung_Opacity-2259.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2259.png,Lung_Opacity/images/Lung_Opacity-2259.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2259.png,Lung_Opacity/masks/Lung_Opacity-2259.png,True,False,299,299,L,143.47,6a9b83c3a3a47e467df9e244505c995a
Lung_Opacity-226.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-226.png,Lung_Opacity/images/Lung_Opacity-226.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-226.png,Lung_Opacity/masks/Lung_Opacity-226.png,True,False,299,299,L,94.52,7da062169a3eebb5a10cc72410771a26
Lung_Opacity-2260.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2260.png,Lung_Opacity/images/Lung_Opacity-2260.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2260.png,Lung_Opacity/masks/Lung_Opacity-2260.png,True,False,299,299,L,132.19,373fb9891ad0ac8378ac1871c7cea390
Lung_Opacity-2261.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2261.png,Lung_Opacity/images/Lung_Opacity-2261.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2261.png,Lung_Opacity/masks/Lung_Opacity-2261.png,True,False,299,299,L,109.14,2b276c35c238f812935837b8a364d306
Lung_Opacity-2262.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2262.png,Lung_Opacity/images/Lung_Opacity-2262.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2262.png,Lung_Opacity/masks/Lung_Opacity-2262.png,True,False,299,299,L,122.32,abb6af70299d50b51a7068711cfa1a49
Lung_Opacity-2263.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2263.png,Lung_Opacity/images/Lung_Opacity-2263.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2263.png,Lung_Opacity/masks/Lung_Opacity-2263.png,True,False,299,299,L,97.52,616d9edc12d8b33369cc82abbd913320
Lung_Opacity-2264.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2264.png,Lung_Opacity/images/Lung_Opacity-2264.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2264.png,Lung_Opacity/masks/Lung_Opacity-2264.png,True,False,299,299,L,93.88,f51271f8561c00e388ac095d5f7aab16
Lung_Opacity-2265.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2265.png,Lung_Opacity/images/Lung_Opacity-2265.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2265.png,Lung_Opacity/masks/Lung_Opacity-2265.png,True,False,299,299,L,110.32,c3799450a5033301bd386b1cadc732bd
Lung_Opacity-2266.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2266.png,Lung_Opacity/images/Lung_Opacity-2266.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2266.png,Lung_Opacity/masks/Lung_Opacity-2266.png,True,False,299,299,L,163.04,117d01c2c211b6a60c985c7eef799b55
Lung_Opacity-2267.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2267.png,Lung_Opacity/images/Lung_Opacity-2267.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2267.png,Lung_Opacity/masks/Lung_Opacity-2267.png,True,False,299,299,L,107.16,109321e701de2da39d3fc69589e0d607
Lung_Opacity-2268.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2268.png,Lung_Opacity/images/Lung_Opacity-2268.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2268.png,Lung_Opacity/masks/Lung_Opacity-2268.png,True,False,299,299,L,137.77,51e65577488dc8c3e4dead830f5b374e
Lung_Opacity-2269.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2269.png,Lung_Opacity/images/Lung_Opacity-2269.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2269.png,Lung_Opacity/masks/Lung_Opacity-2269.png,True,False,299,299,L,102.1,1ff94b806a09cd4c8a4ec50f11756cfb
Lung_Opacity-227.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-227.png,Lung_Opacity/images/Lung_Opacity-227.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-227.png,Lung_Opacity/masks/Lung_Opacity-227.png,True,False,299,299,L,118.6,b5a889541c3386fa105e093177f840e3
Lung_Opacity-2270.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2270.png,Lung_Opacity/images/Lung_Opacity-2270.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2270.png,Lung_Opacity/masks/Lung_Opacity-2270.png,True,False,299,299,L,81.03,ea7f2a7f54f316e54c510662e800f431
Lung_Opacity-2271.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2271.png,Lung_Opacity/images/Lung_Opacity-2271.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2271.png,Lung_Opacity/masks/Lung_Opacity-2271.png,True,False,299,299,L,120.39,5687acfa71499bd4f9a6694e2ab9ee0c
Lung_Opacity-2272.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2272.png,Lung_Opacity/images/Lung_Opacity-2272.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2272.png,Lung_Opacity/masks/Lung_Opacity-2272.png,True,False,299,299,L,115.49,124e232ce5b8a5950f004f599e33d11c
Lung_Opacity-2273.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2273.png,Lung_Opacity/images/Lung_Opacity-2273.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2273.png,Lung_Opacity/masks/Lung_Opacity-2273.png,True,False,299,299,L,105.28,3258116ac77ba4df4cf65ee301e6cf2d
Lung_Opacity-2274.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2274.png,Lung_Opacity/images/Lung_Opacity-2274.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2274.png,Lung_Opacity/masks/Lung_Opacity-2274.png,True,False,299,299,L,148.78,7bcf30f9738cfa2047b25de59f4d8f99
Lung_Opacity-2275.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2275.png,Lung_Opacity/images/Lung_Opacity-2275.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2275.png,Lung_Opacity/masks/Lung_Opacity-2275.png,True,False,299,299,L,128.92,52c4f00bceb404378626385cb83e6401
Lung_Opacity-2276.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2276.png,Lung_Opacity/images/Lung_Opacity-2276.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2276.png,Lung_Opacity/masks/Lung_Opacity-2276.png,True,False,299,299,L,115.45,4420d27b8795197d8c06087d0ee8842e
Lung_Opacity-2277.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2277.png,Lung_Opacity/images/Lung_Opacity-2277.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2277.png,Lung_Opacity/masks/Lung_Opacity-2277.png,True,False,299,299,L,125.46,0beb62a2f56fd6a4275c47f1edec0f53
Lung_Opacity-2278.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2278.png,Lung_Opacity/images/Lung_Opacity-2278.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2278.png,Lung_Opacity/masks/Lung_Opacity-2278.png,True,False,299,299,L,116.01,7c9a76cb0bfd6e3d72a6622c51ecb774
Lung_Opacity-2279.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2279.png,Lung_Opacity/images/Lung_Opacity-2279.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2279.png,Lung_Opacity/masks/Lung_Opacity-2279.png,True,False,299,299,L,142.9,32d217adfc9432b07e025918b4456d65
Lung_Opacity-228.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-228.png,Lung_Opacity/images/Lung_Opacity-228.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-228.png,Lung_Opacity/masks/Lung_Opacity-228.png,True,False,299,299,L,101.23,32a6e2fa5ca16bf1d73050a0d3ede24a
Lung_Opacity-2280.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2280.png,Lung_Opacity/images/Lung_Opacity-2280.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2280.png,Lung_Opacity/masks/Lung_Opacity-2280.png,True,False,299,299,L,76.46,e5bfd6325823bd9012b196aca162f40f
Lung_Opacity-2281.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2281.png,Lung_Opacity/images/Lung_Opacity-2281.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2281.png,Lung_Opacity/masks/Lung_Opacity-2281.png,True,False,299,299,L,132.82,b4ba64205d554d431efa44696155528a
Lung_Opacity-2282.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2282.png,Lung_Opacity/images/Lung_Opacity-2282.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2282.png,Lung_Opacity/masks/Lung_Opacity-2282.png,True,False,299,299,L,158.02,fc2e708f0ddf8417f75c0a94748f1995
Lung_Opacity-2283.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2283.png,Lung_Opacity/images/Lung_Opacity-2283.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2283.png,Lung_Opacity/masks/Lung_Opacity-2283.png,True,False,299,299,L,69.75,8291cfff039d17a686789559b2721c8b
Lung_Opacity-2284.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2284.png,Lung_Opacity/images/Lung_Opacity-2284.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2284.png,Lung_Opacity/masks/Lung_Opacity-2284.png,True,False,299,299,L,114.65,c124074de2044417a72e73bc99936878
Lung_Opacity-2285.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2285.png,Lung_Opacity/images/Lung_Opacity-2285.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2285.png,Lung_Opacity/masks/Lung_Opacity-2285.png,True,False,299,299,L,123.1,615097879261e9004ffa682f99d3cfa3
Lung_Opacity-2286.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2286.png,Lung_Opacity/images/Lung_Opacity-2286.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2286.png,Lung_Opacity/masks/Lung_Opacity-2286.png,True,False,299,299,L,167.28,672243cf389035c638c34d52b5f9b6fe
Lung_Opacity-2287.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2287.png,Lung_Opacity/images/Lung_Opacity-2287.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2287.png,Lung_Opacity/masks/Lung_Opacity-2287.png,True,False,299,299,L,136.47,9aff832eedd50a79c60171922b8761c3
Lung_Opacity-2288.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2288.png,Lung_Opacity/images/Lung_Opacity-2288.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2288.png,Lung_Opacity/masks/Lung_Opacity-2288.png,True,False,299,299,L,105.75,78f0cc95ba11292df0d83f0dab4cd290
Lung_Opacity-2289.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2289.png,Lung_Opacity/images/Lung_Opacity-2289.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2289.png,Lung_Opacity/masks/Lung_Opacity-2289.png,True,False,299,299,L,121.36,e53dcd508d396bd40e09efd0162ba18b
Lung_Opacity-229.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-229.png,Lung_Opacity/images/Lung_Opacity-229.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-229.png,Lung_Opacity/masks/Lung_Opacity-229.png,True,False,299,299,L,129.14,c4c87d05b9f1e94e06c8e6beed41c22e
Lung_Opacity-2290.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2290.png,Lung_Opacity/images/Lung_Opacity-2290.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2290.png,Lung_Opacity/masks/Lung_Opacity-2290.png,True,False,299,299,L,172.85,2eff544ae7458df1e66543a3cc6229b4
Lung_Opacity-2291.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2291.png,Lung_Opacity/images/Lung_Opacity-2291.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2291.png,Lung_Opacity/masks/Lung_Opacity-2291.png,True,False,299,299,L,132.47,4bbb6a7c51327144de98554fd1e54e2c
Lung_Opacity-2292.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2292.png,Lung_Opacity/images/Lung_Opacity-2292.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2292.png,Lung_Opacity/masks/Lung_Opacity-2292.png,True,False,299,299,L,168.98,80250342acc4c4b3eada5e0e92552570
Lung_Opacity-2293.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2293.png,Lung_Opacity/images/Lung_Opacity-2293.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2293.png,Lung_Opacity/masks/Lung_Opacity-2293.png,True,False,299,299,L,110.49,2d460e066415e632c88a105cdcfc8d1d
Lung_Opacity-2294.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2294.png,Lung_Opacity/images/Lung_Opacity-2294.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2294.png,Lung_Opacity/masks/Lung_Opacity-2294.png,True,False,299,299,L,128.51,0cdfd145abaf88e9a6251eb5e0bafb2a
Lung_Opacity-2295.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2295.png,Lung_Opacity/images/Lung_Opacity-2295.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2295.png,Lung_Opacity/masks/Lung_Opacity-2295.png,True,False,299,299,L,156.05,2668b55b13e043ae499c6d35342c3158
Lung_Opacity-2296.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2296.png,Lung_Opacity/images/Lung_Opacity-2296.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2296.png,Lung_Opacity/masks/Lung_Opacity-2296.png,True,False,299,299,L,176.5,70bbc9b65b0d8ab58f8cc3bf56264eb3
Lung_Opacity-2297.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2297.png,Lung_Opacity/images/Lung_Opacity-2297.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2297.png,Lung_Opacity/masks/Lung_Opacity-2297.png,True,False,299,299,L,124.79,090dfab70359e4f27b18fe7847e38727
Lung_Opacity-2298.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2298.png,Lung_Opacity/images/Lung_Opacity-2298.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2298.png,Lung_Opacity/masks/Lung_Opacity-2298.png,True,False,299,299,L,110.93,1f04abda8377a04028594bd4fb9012bb
Lung_Opacity-2299.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2299.png,Lung_Opacity/images/Lung_Opacity-2299.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2299.png,Lung_Opacity/masks/Lung_Opacity-2299.png,True,False,299,299,L,108.74,e48f9e2115f19f22eb99b84577b1e892
Lung_Opacity-23.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-23.png,Lung_Opacity/images/Lung_Opacity-23.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-23.png,Lung_Opacity/masks/Lung_Opacity-23.png,True,False,299,299,L,124.27,339700a3e525fdf28cab0e3793175ab0
Lung_Opacity-230.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-230.png,Lung_Opacity/images/Lung_Opacity-230.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-230.png,Lung_Opacity/masks/Lung_Opacity-230.png,True,False,299,299,L,153.24,83209d67f40abdfb3645df0a6746c4da
Lung_Opacity-2300.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2300.png,Lung_Opacity/images/Lung_Opacity-2300.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2300.png,Lung_Opacity/masks/Lung_Opacity-2300.png,True,False,299,299,L,122.48,e70d2712147aa68cb8b2ac1598fb7468
Lung_Opacity-2301.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2301.png,Lung_Opacity/images/Lung_Opacity-2301.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2301.png,Lung_Opacity/masks/Lung_Opacity-2301.png,True,False,299,299,L,119.14,b2b5ee6377dd8950fc318ab790681302
Lung_Opacity-2302.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2302.png,Lung_Opacity/images/Lung_Opacity-2302.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2302.png,Lung_Opacity/masks/Lung_Opacity-2302.png,True,False,299,299,L,130.19,8b436e703ec202f401ff80b77a061492
Lung_Opacity-2303.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2303.png,Lung_Opacity/images/Lung_Opacity-2303.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2303.png,Lung_Opacity/masks/Lung_Opacity-2303.png,True,False,299,299,L,113.88,bced421bdc3984bc5fe45423d16e24d5
Lung_Opacity-2304.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2304.png,Lung_Opacity/images/Lung_Opacity-2304.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2304.png,Lung_Opacity/masks/Lung_Opacity-2304.png,True,False,299,299,L,156.62,990cc437f8acd71e557cfb6990e6d6f0
Lung_Opacity-2305.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2305.png,Lung_Opacity/images/Lung_Opacity-2305.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2305.png,Lung_Opacity/masks/Lung_Opacity-2305.png,True,False,299,299,L,114.78,45272c7cd2ef4af3f371788dc5073649
Lung_Opacity-2306.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2306.png,Lung_Opacity/images/Lung_Opacity-2306.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2306.png,Lung_Opacity/masks/Lung_Opacity-2306.png,True,False,299,299,L,141.64,bd82c3632a921935b062f7fedf72af19
Lung_Opacity-2307.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2307.png,Lung_Opacity/images/Lung_Opacity-2307.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2307.png,Lung_Opacity/masks/Lung_Opacity-2307.png,True,False,299,299,L,114.3,4f26a972a6c02ca8885c0d89eeca4a38
Lung_Opacity-2308.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2308.png,Lung_Opacity/images/Lung_Opacity-2308.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2308.png,Lung_Opacity/masks/Lung_Opacity-2308.png,True,False,299,299,L,112.7,b5c9382d06043712434615bf8d0a0f5d
Lung_Opacity-2309.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2309.png,Lung_Opacity/images/Lung_Opacity-2309.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2309.png,Lung_Opacity/masks/Lung_Opacity-2309.png,True,False,299,299,L,116.19,49864463b881804ba02686ab668a613c
Lung_Opacity-231.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-231.png,Lung_Opacity/images/Lung_Opacity-231.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-231.png,Lung_Opacity/masks/Lung_Opacity-231.png,True,False,299,299,L,158.11,d27ae3059cbec0cf427844beda4631f5
Lung_Opacity-2310.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2310.png,Lung_Opacity/images/Lung_Opacity-2310.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2310.png,Lung_Opacity/masks/Lung_Opacity-2310.png,True,False,299,299,L,110.88,8db106556fc903c5e9b18f8ffdd75313
Lung_Opacity-2311.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2311.png,Lung_Opacity/images/Lung_Opacity-2311.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2311.png,Lung_Opacity/masks/Lung_Opacity-2311.png,True,False,299,299,L,158.96,7280ff637dbe73e7e0f4b709b38f12f2
Lung_Opacity-2312.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2312.png,Lung_Opacity/images/Lung_Opacity-2312.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2312.png,Lung_Opacity/masks/Lung_Opacity-2312.png,True,False,299,299,L,121.64,52ec0368ef191fe07b12f0ab9e173a5a
Lung_Opacity-2313.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2313.png,Lung_Opacity/images/Lung_Opacity-2313.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2313.png,Lung_Opacity/masks/Lung_Opacity-2313.png,True,False,299,299,L,118.74,34e294a69e14ae9ae66b3afb668ceaa9
Lung_Opacity-2314.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2314.png,Lung_Opacity/images/Lung_Opacity-2314.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2314.png,Lung_Opacity/masks/Lung_Opacity-2314.png,True,False,299,299,L,125.45,2abaf6d74827d554eb38edbac59126c1
Lung_Opacity-2315.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2315.png,Lung_Opacity/images/Lung_Opacity-2315.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2315.png,Lung_Opacity/masks/Lung_Opacity-2315.png,True,False,299,299,L,114.67,1505d51fe2b22e9dcc3f6ced92835520
Lung_Opacity-2316.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2316.png,Lung_Opacity/images/Lung_Opacity-2316.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2316.png,Lung_Opacity/masks/Lung_Opacity-2316.png,True,False,299,299,L,108.43,56bc8410a860b28995e10a70b5a0f932
Lung_Opacity-2317.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2317.png,Lung_Opacity/images/Lung_Opacity-2317.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2317.png,Lung_Opacity/masks/Lung_Opacity-2317.png,True,False,299,299,L,149.42,4f7abd7e3b0ed2b58043ff74aac176c3
Lung_Opacity-2318.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2318.png,Lung_Opacity/images/Lung_Opacity-2318.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2318.png,Lung_Opacity/masks/Lung_Opacity-2318.png,True,False,299,299,L,111.42,c25f55ff94bffa169788ac4bfdc83770
Lung_Opacity-2319.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2319.png,Lung_Opacity/images/Lung_Opacity-2319.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2319.png,Lung_Opacity/masks/Lung_Opacity-2319.png,True,False,299,299,L,147.08,9048175e7c6351c142359ef9edf16d45
Lung_Opacity-232.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-232.png,Lung_Opacity/images/Lung_Opacity-232.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-232.png,Lung_Opacity/masks/Lung_Opacity-232.png,True,False,299,299,L,137.49,84ec6e48519bc1334ec72391aa96dd04
Lung_Opacity-2320.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2320.png,Lung_Opacity/images/Lung_Opacity-2320.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2320.png,Lung_Opacity/masks/Lung_Opacity-2320.png,True,False,299,299,L,171.5,12803ae1752fdfe9ae034944c8ced4f2
Lung_Opacity-2321.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2321.png,Lung_Opacity/images/Lung_Opacity-2321.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2321.png,Lung_Opacity/masks/Lung_Opacity-2321.png,True,False,299,299,L,119.01,96fba76abf42c93956203f7be89067d8
Lung_Opacity-2322.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2322.png,Lung_Opacity/images/Lung_Opacity-2322.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2322.png,Lung_Opacity/masks/Lung_Opacity-2322.png,True,False,299,299,L,121.14,e53ae096404525b1121910aba2aa18b5
Lung_Opacity-2323.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2323.png,Lung_Opacity/images/Lung_Opacity-2323.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2323.png,Lung_Opacity/masks/Lung_Opacity-2323.png,True,False,299,299,L,109.03,164e24c2ddb44b9e98e27148e6dbe9d1
Lung_Opacity-2324.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2324.png,Lung_Opacity/images/Lung_Opacity-2324.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2324.png,Lung_Opacity/masks/Lung_Opacity-2324.png,True,False,299,299,L,128.39,f5f1164bd5c7fa8153add31e4200d97c
Lung_Opacity-2325.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2325.png,Lung_Opacity/images/Lung_Opacity-2325.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2325.png,Lung_Opacity/masks/Lung_Opacity-2325.png,True,False,299,299,L,119.79,9bc38f5d443f6ba7f9916007d8e0c64e
Lung_Opacity-2326.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2326.png,Lung_Opacity/images/Lung_Opacity-2326.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2326.png,Lung_Opacity/masks/Lung_Opacity-2326.png,True,False,299,299,L,135.23,1e57b5d8633c07f261904183c2599b23
Lung_Opacity-2327.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2327.png,Lung_Opacity/images/Lung_Opacity-2327.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2327.png,Lung_Opacity/masks/Lung_Opacity-2327.png,True,False,299,299,L,112.49,15c98af66c78c59221841aef012900eb
Lung_Opacity-2328.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2328.png,Lung_Opacity/images/Lung_Opacity-2328.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2328.png,Lung_Opacity/masks/Lung_Opacity-2328.png,True,False,299,299,L,117.54,fa0b74c660b3a563b02bfc9276bc8ed7
Lung_Opacity-2329.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2329.png,Lung_Opacity/images/Lung_Opacity-2329.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2329.png,Lung_Opacity/masks/Lung_Opacity-2329.png,True,False,299,299,L,115.84,0f6fb0c4f5dc95be75c0dab778e3c101
Lung_Opacity-233.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-233.png,Lung_Opacity/images/Lung_Opacity-233.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-233.png,Lung_Opacity/masks/Lung_Opacity-233.png,True,False,299,299,L,91.02,4d0718edeed0111eac18c85019cd8334
Lung_Opacity-2330.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2330.png,Lung_Opacity/images/Lung_Opacity-2330.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2330.png,Lung_Opacity/masks/Lung_Opacity-2330.png,True,False,299,299,L,112.67,c980ce0f4253d82c67bd6d4f4efb6524
Lung_Opacity-2331.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2331.png,Lung_Opacity/images/Lung_Opacity-2331.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2331.png,Lung_Opacity/masks/Lung_Opacity-2331.png,True,False,299,299,L,123.19,16606324149275a7666a61d26d6d32ac
Lung_Opacity-2332.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2332.png,Lung_Opacity/images/Lung_Opacity-2332.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2332.png,Lung_Opacity/masks/Lung_Opacity-2332.png,True,False,299,299,L,116.5,3007dfb02589addca1d3c350cad8dbfe
Lung_Opacity-2333.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2333.png,Lung_Opacity/images/Lung_Opacity-2333.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2333.png,Lung_Opacity/masks/Lung_Opacity-2333.png,True,False,299,299,L,110.16,e14f4037b8a64a96823572a6c45105a1
Lung_Opacity-2334.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2334.png,Lung_Opacity/images/Lung_Opacity-2334.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2334.png,Lung_Opacity/masks/Lung_Opacity-2334.png,True,False,299,299,L,111.01,47f68bafc96c237584cf804d2d62213c
Lung_Opacity-2335.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2335.png,Lung_Opacity/images/Lung_Opacity-2335.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2335.png,Lung_Opacity/masks/Lung_Opacity-2335.png,True,False,299,299,L,84.54,be33878a10a335d5586c49b1bef7602c
Lung_Opacity-2336.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2336.png,Lung_Opacity/images/Lung_Opacity-2336.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2336.png,Lung_Opacity/masks/Lung_Opacity-2336.png,True,False,299,299,L,150.55,10cc962e314c55283ee579173d8810c8
Lung_Opacity-2337.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2337.png,Lung_Opacity/images/Lung_Opacity-2337.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2337.png,Lung_Opacity/masks/Lung_Opacity-2337.png,True,False,299,299,L,164.19,2b21186d2c58a33fb4edd22ec2e4ae60
Lung_Opacity-2338.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2338.png,Lung_Opacity/images/Lung_Opacity-2338.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2338.png,Lung_Opacity/masks/Lung_Opacity-2338.png,True,False,299,299,L,124.68,a726d0fd77a02906d45da0602f63f57e
Lung_Opacity-2339.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2339.png,Lung_Opacity/images/Lung_Opacity-2339.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2339.png,Lung_Opacity/masks/Lung_Opacity-2339.png,True,False,299,299,L,86.31,867df51a5b9aba1040a39b6ec650cbad
Lung_Opacity-234.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-234.png,Lung_Opacity/images/Lung_Opacity-234.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-234.png,Lung_Opacity/masks/Lung_Opacity-234.png,True,False,299,299,L,107.74,9450573eb55329db92fba34294b58461
Lung_Opacity-2340.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2340.png,Lung_Opacity/images/Lung_Opacity-2340.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2340.png,Lung_Opacity/masks/Lung_Opacity-2340.png,True,False,299,299,L,172.79,d290892b407746abdd6859d1e9cde738
Lung_Opacity-2341.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2341.png,Lung_Opacity/images/Lung_Opacity-2341.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2341.png,Lung_Opacity/masks/Lung_Opacity-2341.png,True,False,299,299,L,153.49,dfead1c603edf143090adfe683d34bf1
Lung_Opacity-2342.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2342.png,Lung_Opacity/images/Lung_Opacity-2342.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2342.png,Lung_Opacity/masks/Lung_Opacity-2342.png,True,False,299,299,L,111.88,b5477c145a2930244a41468b38ba23d3
Lung_Opacity-2343.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2343.png,Lung_Opacity/images/Lung_Opacity-2343.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2343.png,Lung_Opacity/masks/Lung_Opacity-2343.png,True,False,299,299,L,112.38,d15a24f36a3891d4d7429cd3045c390e
Lung_Opacity-2344.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2344.png,Lung_Opacity/images/Lung_Opacity-2344.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2344.png,Lung_Opacity/masks/Lung_Opacity-2344.png,True,False,299,299,L,135.85,11613dde3f33383ddec4b51b16a82afe
Lung_Opacity-2345.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2345.png,Lung_Opacity/images/Lung_Opacity-2345.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2345.png,Lung_Opacity/masks/Lung_Opacity-2345.png,True,False,299,299,L,123.74,95e0b4a39a02654c17075d77f66490d0
Lung_Opacity-2346.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2346.png,Lung_Opacity/images/Lung_Opacity-2346.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2346.png,Lung_Opacity/masks/Lung_Opacity-2346.png,True,False,299,299,L,125.12,ca52c35eb50713bd25459cc52e7f7cd1
Lung_Opacity-2347.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2347.png,Lung_Opacity/images/Lung_Opacity-2347.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2347.png,Lung_Opacity/masks/Lung_Opacity-2347.png,True,False,299,299,L,110.27,9db68155e1127149af4751b0347afbdd
Lung_Opacity-2348.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2348.png,Lung_Opacity/images/Lung_Opacity-2348.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2348.png,Lung_Opacity/masks/Lung_Opacity-2348.png,True,False,299,299,L,113.95,e71bf37c019d722afbed00162edaafae
Lung_Opacity-2349.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2349.png,Lung_Opacity/images/Lung_Opacity-2349.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2349.png,Lung_Opacity/masks/Lung_Opacity-2349.png,True,False,299,299,L,119.28,a90d2bdc2765a1e275e1e0852d94e5c1
Lung_Opacity-235.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-235.png,Lung_Opacity/images/Lung_Opacity-235.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-235.png,Lung_Opacity/masks/Lung_Opacity-235.png,True,False,299,299,L,111.29,93529c1fabb89732db3990d9d70b6fc1
Lung_Opacity-2350.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2350.png,Lung_Opacity/images/Lung_Opacity-2350.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2350.png,Lung_Opacity/masks/Lung_Opacity-2350.png,True,False,299,299,L,168.04,271b0e5d6ffe78686de6e0b3cea709f0
Lung_Opacity-2351.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2351.png,Lung_Opacity/images/Lung_Opacity-2351.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2351.png,Lung_Opacity/masks/Lung_Opacity-2351.png,True,False,299,299,L,110.35,c916e0868f72f8668e67f0358a5c19c7
Lung_Opacity-2352.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2352.png,Lung_Opacity/images/Lung_Opacity-2352.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2352.png,Lung_Opacity/masks/Lung_Opacity-2352.png,True,False,299,299,L,138.08,4da7886859170164266cc2124d2588f1
Lung_Opacity-2353.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2353.png,Lung_Opacity/images/Lung_Opacity-2353.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2353.png,Lung_Opacity/masks/Lung_Opacity-2353.png,True,False,299,299,L,108.27,bd097859c1da1414e2a12be00804b0df
Lung_Opacity-2354.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2354.png,Lung_Opacity/images/Lung_Opacity-2354.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2354.png,Lung_Opacity/masks/Lung_Opacity-2354.png,True,False,299,299,L,157.94,00a68bdd5467f0f58590732ffc2130a2
Lung_Opacity-2355.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2355.png,Lung_Opacity/images/Lung_Opacity-2355.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2355.png,Lung_Opacity/masks/Lung_Opacity-2355.png,True,False,299,299,L,118.84,6390017a503925f882804a26ca4e2712
Lung_Opacity-2356.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2356.png,Lung_Opacity/images/Lung_Opacity-2356.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2356.png,Lung_Opacity/masks/Lung_Opacity-2356.png,True,False,299,299,L,122.32,f0a1df843ab4cde872f9782ff5647af1
Lung_Opacity-2357.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2357.png,Lung_Opacity/images/Lung_Opacity-2357.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2357.png,Lung_Opacity/masks/Lung_Opacity-2357.png,True,False,299,299,L,166.67,45d7736e07e0c1509c62a9be697b7bab
Lung_Opacity-2358.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2358.png,Lung_Opacity/images/Lung_Opacity-2358.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2358.png,Lung_Opacity/masks/Lung_Opacity-2358.png,True,False,299,299,L,98.41,a3b3564948f9a03be4b58e5917a2500c
Lung_Opacity-2359.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2359.png,Lung_Opacity/images/Lung_Opacity-2359.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2359.png,Lung_Opacity/masks/Lung_Opacity-2359.png,True,False,299,299,L,126.78,149a11013797c66c95d344bf331d4573
Lung_Opacity-236.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-236.png,Lung_Opacity/images/Lung_Opacity-236.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-236.png,Lung_Opacity/masks/Lung_Opacity-236.png,True,False,299,299,L,159.74,174b23961d7fbe9fe8438bef74588ede
Lung_Opacity-2360.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2360.png,Lung_Opacity/images/Lung_Opacity-2360.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2360.png,Lung_Opacity/masks/Lung_Opacity-2360.png,True,False,299,299,L,110.81,90cefce38587dd9070e3874182b00f4a
Lung_Opacity-2361.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2361.png,Lung_Opacity/images/Lung_Opacity-2361.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2361.png,Lung_Opacity/masks/Lung_Opacity-2361.png,True,False,299,299,L,139.79,748de92428b1b8ce2c1bd2221ff40d4a
Lung_Opacity-2362.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2362.png,Lung_Opacity/images/Lung_Opacity-2362.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2362.png,Lung_Opacity/masks/Lung_Opacity-2362.png,True,False,299,299,L,125.43,e6efdbbf1d3ba720f85a16a0b8ceacc6
Lung_Opacity-2363.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2363.png,Lung_Opacity/images/Lung_Opacity-2363.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2363.png,Lung_Opacity/masks/Lung_Opacity-2363.png,True,False,299,299,L,111.44,7f5be9e04d84c2b53f3556c56f43d056
Lung_Opacity-2364.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2364.png,Lung_Opacity/images/Lung_Opacity-2364.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2364.png,Lung_Opacity/masks/Lung_Opacity-2364.png,True,False,299,299,L,110.03,67f5d8cfadd371164fb9a382b58b75db
Lung_Opacity-2365.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2365.png,Lung_Opacity/images/Lung_Opacity-2365.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2365.png,Lung_Opacity/masks/Lung_Opacity-2365.png,True,False,299,299,L,178.16,5c8739f82533dc1cbeae452796802002
Lung_Opacity-2366.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2366.png,Lung_Opacity/images/Lung_Opacity-2366.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2366.png,Lung_Opacity/masks/Lung_Opacity-2366.png,True,False,299,299,L,155.38,f5d999e7ea46f535576a4dc38fcc0883
Lung_Opacity-2367.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2367.png,Lung_Opacity/images/Lung_Opacity-2367.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2367.png,Lung_Opacity/masks/Lung_Opacity-2367.png,True,False,299,299,L,156.42,42a7a611599254c326237bd83fca2008
Lung_Opacity-2368.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2368.png,Lung_Opacity/images/Lung_Opacity-2368.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2368.png,Lung_Opacity/masks/Lung_Opacity-2368.png,True,False,299,299,L,104.79,64aed7d0bbbc711f7f014c46aab38422
Lung_Opacity-2369.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2369.png,Lung_Opacity/images/Lung_Opacity-2369.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2369.png,Lung_Opacity/masks/Lung_Opacity-2369.png,True,False,299,299,L,137.66,352e1b5088e98f6b9631b03da4d706ce
Lung_Opacity-237.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-237.png,Lung_Opacity/images/Lung_Opacity-237.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-237.png,Lung_Opacity/masks/Lung_Opacity-237.png,True,False,299,299,L,105.13,779809b2a7138d420516413272834c37
Lung_Opacity-2370.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2370.png,Lung_Opacity/images/Lung_Opacity-2370.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2370.png,Lung_Opacity/masks/Lung_Opacity-2370.png,True,False,299,299,L,140.07,e12efa51057e8244d83e543d6575944b
Lung_Opacity-2371.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2371.png,Lung_Opacity/images/Lung_Opacity-2371.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2371.png,Lung_Opacity/masks/Lung_Opacity-2371.png,True,False,299,299,L,168.69,5dac91ebfb07a0b15b1c19a8cea17ca8
Lung_Opacity-2372.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2372.png,Lung_Opacity/images/Lung_Opacity-2372.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2372.png,Lung_Opacity/masks/Lung_Opacity-2372.png,True,False,299,299,L,113.39,31b710aee7002f959ab0aef7cff851b9
Lung_Opacity-2373.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2373.png,Lung_Opacity/images/Lung_Opacity-2373.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2373.png,Lung_Opacity/masks/Lung_Opacity-2373.png,True,False,299,299,L,110.18,ab5f9e6b437e8e53af3769227bafeb72
Lung_Opacity-2374.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2374.png,Lung_Opacity/images/Lung_Opacity-2374.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2374.png,Lung_Opacity/masks/Lung_Opacity-2374.png,True,False,299,299,L,110.81,309de728c7644d612253488e85ca2274
Lung_Opacity-2375.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2375.png,Lung_Opacity/images/Lung_Opacity-2375.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2375.png,Lung_Opacity/masks/Lung_Opacity-2375.png,True,False,299,299,L,183.71,417d9c77e141ea84866122a48c11b3d9
Lung_Opacity-2376.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2376.png,Lung_Opacity/images/Lung_Opacity-2376.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2376.png,Lung_Opacity/masks/Lung_Opacity-2376.png,True,False,299,299,L,81.27,f001d62f8858d9ec71d8ee5fb3407645
Lung_Opacity-2377.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2377.png,Lung_Opacity/images/Lung_Opacity-2377.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2377.png,Lung_Opacity/masks/Lung_Opacity-2377.png,True,False,299,299,L,133.26,4e0ca1a4c73c179a8dd77c3bd36064fa
Lung_Opacity-2378.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2378.png,Lung_Opacity/images/Lung_Opacity-2378.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2378.png,Lung_Opacity/masks/Lung_Opacity-2378.png,True,False,299,299,L,167.57,62738938e61f8444fbd217f17705738a
Lung_Opacity-2379.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2379.png,Lung_Opacity/images/Lung_Opacity-2379.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2379.png,Lung_Opacity/masks/Lung_Opacity-2379.png,True,False,299,299,L,119.18,9fb2a9a9611307d47a7371c2f6227c9e
Lung_Opacity-238.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-238.png,Lung_Opacity/images/Lung_Opacity-238.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-238.png,Lung_Opacity/masks/Lung_Opacity-238.png,True,False,299,299,L,114.02,0225571d2f857c88355963ddf00a22eb
Lung_Opacity-2380.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2380.png,Lung_Opacity/images/Lung_Opacity-2380.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2380.png,Lung_Opacity/masks/Lung_Opacity-2380.png,True,False,299,299,L,110.61,144c69ec4e57001dcf252b47895f7235
Lung_Opacity-2381.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2381.png,Lung_Opacity/images/Lung_Opacity-2381.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2381.png,Lung_Opacity/masks/Lung_Opacity-2381.png,True,False,299,299,L,138.5,4929262332ca90e0429c636e0ab2995c
Lung_Opacity-2382.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2382.png,Lung_Opacity/images/Lung_Opacity-2382.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2382.png,Lung_Opacity/masks/Lung_Opacity-2382.png,True,False,299,299,L,135.58,c748247ce25ae776f6c18b058eb5e1a6
Lung_Opacity-2383.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2383.png,Lung_Opacity/images/Lung_Opacity-2383.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2383.png,Lung_Opacity/masks/Lung_Opacity-2383.png,True,False,299,299,L,149.48,e5bf1ee8148d55f09e5cfccabe714a6f
Lung_Opacity-2384.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2384.png,Lung_Opacity/images/Lung_Opacity-2384.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2384.png,Lung_Opacity/masks/Lung_Opacity-2384.png,True,False,299,299,L,112.8,fc325564536529dbfdb39b6819522587
Lung_Opacity-2385.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2385.png,Lung_Opacity/images/Lung_Opacity-2385.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2385.png,Lung_Opacity/masks/Lung_Opacity-2385.png,True,False,299,299,L,89.12,9b39d19496e9ed9d23219bdba67a38b1
Lung_Opacity-2386.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2386.png,Lung_Opacity/images/Lung_Opacity-2386.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2386.png,Lung_Opacity/masks/Lung_Opacity-2386.png,True,False,299,299,L,111.3,2e3e2013a8b3361135b09078553b82da
Lung_Opacity-2387.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2387.png,Lung_Opacity/images/Lung_Opacity-2387.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2387.png,Lung_Opacity/masks/Lung_Opacity-2387.png,True,False,299,299,L,127.58,74d965942bce672fb2487e83d2de988a
Lung_Opacity-2388.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2388.png,Lung_Opacity/images/Lung_Opacity-2388.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2388.png,Lung_Opacity/masks/Lung_Opacity-2388.png,True,False,299,299,L,95.99,58e9e9f6af0d9c7cf3547b8268d49e4d
Lung_Opacity-2389.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2389.png,Lung_Opacity/images/Lung_Opacity-2389.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2389.png,Lung_Opacity/masks/Lung_Opacity-2389.png,True,False,299,299,L,115.58,20967bfa8a6ec701c5dbfc29465edd02
Lung_Opacity-239.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-239.png,Lung_Opacity/images/Lung_Opacity-239.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-239.png,Lung_Opacity/masks/Lung_Opacity-239.png,True,False,299,299,L,114.21,3389a63f2ea87e48e2d056cb491d3815
Lung_Opacity-2390.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2390.png,Lung_Opacity/images/Lung_Opacity-2390.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2390.png,Lung_Opacity/masks/Lung_Opacity-2390.png,True,False,299,299,L,172.07,a653fbff626a849f6e6985085ae819d1
Lung_Opacity-2391.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2391.png,Lung_Opacity/images/Lung_Opacity-2391.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2391.png,Lung_Opacity/masks/Lung_Opacity-2391.png,True,False,299,299,L,116.26,7e3dacdd9097d98ac521fc012d2a84ff
Lung_Opacity-2392.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2392.png,Lung_Opacity/images/Lung_Opacity-2392.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2392.png,Lung_Opacity/masks/Lung_Opacity-2392.png,True,False,299,299,L,72.64,554e56549bf8c325da2295269531d68c
Lung_Opacity-2393.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2393.png,Lung_Opacity/images/Lung_Opacity-2393.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2393.png,Lung_Opacity/masks/Lung_Opacity-2393.png,True,False,299,299,L,175.18,bf6b60cf4fa9f15681579e4e9ca5f4df
Lung_Opacity-2394.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2394.png,Lung_Opacity/images/Lung_Opacity-2394.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2394.png,Lung_Opacity/masks/Lung_Opacity-2394.png,True,False,299,299,L,146.34,e067f1e850c49ade82b0c0a7d4cea7e4
Lung_Opacity-2395.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2395.png,Lung_Opacity/images/Lung_Opacity-2395.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2395.png,Lung_Opacity/masks/Lung_Opacity-2395.png,True,False,299,299,L,124.34,1212f919687c0bb5f0497d5964432128
Lung_Opacity-2396.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2396.png,Lung_Opacity/images/Lung_Opacity-2396.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2396.png,Lung_Opacity/masks/Lung_Opacity-2396.png,True,False,299,299,L,164.76,349ae91b4cb2e4871e1534abdba9c19b
Lung_Opacity-2397.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2397.png,Lung_Opacity/images/Lung_Opacity-2397.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2397.png,Lung_Opacity/masks/Lung_Opacity-2397.png,True,False,299,299,L,104.32,934f85aee9e3aa95ec8bbc2a50dababc
Lung_Opacity-2398.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2398.png,Lung_Opacity/images/Lung_Opacity-2398.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2398.png,Lung_Opacity/masks/Lung_Opacity-2398.png,True,False,299,299,L,164.19,81c0d8d086111973e13e02646c9ce599
Lung_Opacity-2399.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2399.png,Lung_Opacity/images/Lung_Opacity-2399.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2399.png,Lung_Opacity/masks/Lung_Opacity-2399.png,True,False,299,299,L,100.9,0ed4eeea0e67ed1c4b2375f8d9d091a8
Lung_Opacity-24.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-24.png,Lung_Opacity/images/Lung_Opacity-24.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-24.png,Lung_Opacity/masks/Lung_Opacity-24.png,True,False,299,299,L,139.48,0dfb3fab1973228925ba924a9df85717
Lung_Opacity-240.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-240.png,Lung_Opacity/images/Lung_Opacity-240.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-240.png,Lung_Opacity/masks/Lung_Opacity-240.png,True,False,299,299,L,168.5,ae0ace89e5dc811b184cbd0f0a939f2a
Lung_Opacity-2400.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2400.png,Lung_Opacity/images/Lung_Opacity-2400.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2400.png,Lung_Opacity/masks/Lung_Opacity-2400.png,True,False,299,299,L,105.67,1326bc06129725d6147cafad1bfe553f
Lung_Opacity-2401.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2401.png,Lung_Opacity/images/Lung_Opacity-2401.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2401.png,Lung_Opacity/masks/Lung_Opacity-2401.png,True,False,299,299,L,129.31,dbc61a99b66fa4fcd1126cbe9a0468e8
Lung_Opacity-2402.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2402.png,Lung_Opacity/images/Lung_Opacity-2402.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2402.png,Lung_Opacity/masks/Lung_Opacity-2402.png,True,False,299,299,L,103.52,0491f09ad63c8f926aeac51966d061fe
Lung_Opacity-2403.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2403.png,Lung_Opacity/images/Lung_Opacity-2403.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2403.png,Lung_Opacity/masks/Lung_Opacity-2403.png,True,False,299,299,L,157.16,3bcbf83f084dcd585a2144cf26ed9478
Lung_Opacity-2404.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2404.png,Lung_Opacity/images/Lung_Opacity-2404.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2404.png,Lung_Opacity/masks/Lung_Opacity-2404.png,True,False,299,299,L,84.0,3def200d6d4762ff03557684b7871560
Lung_Opacity-2405.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2405.png,Lung_Opacity/images/Lung_Opacity-2405.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2405.png,Lung_Opacity/masks/Lung_Opacity-2405.png,True,False,299,299,L,96.65,ed37bd286469b79cc2ab272bdad7c60c
Lung_Opacity-2406.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2406.png,Lung_Opacity/images/Lung_Opacity-2406.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2406.png,Lung_Opacity/masks/Lung_Opacity-2406.png,True,False,299,299,L,107.02,745aada15e5a8c7c4ba9f42df258a4ce
Lung_Opacity-2407.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2407.png,Lung_Opacity/images/Lung_Opacity-2407.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2407.png,Lung_Opacity/masks/Lung_Opacity-2407.png,True,False,299,299,L,152.55,3a80c43af9991cef18be0dce6b668cc1
Lung_Opacity-2408.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2408.png,Lung_Opacity/images/Lung_Opacity-2408.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2408.png,Lung_Opacity/masks/Lung_Opacity-2408.png,True,False,299,299,L,160.77,30716540812f1bdc6aaa15151738219c
Lung_Opacity-2409.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2409.png,Lung_Opacity/images/Lung_Opacity-2409.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2409.png,Lung_Opacity/masks/Lung_Opacity-2409.png,True,False,299,299,L,119.43,2add32b6bd0145feeea8d99ef3bae97a
Lung_Opacity-241.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-241.png,Lung_Opacity/images/Lung_Opacity-241.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-241.png,Lung_Opacity/masks/Lung_Opacity-241.png,True,False,299,299,L,114.67,8c8fbe31dda97b3248cea607b9513114
Lung_Opacity-2410.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2410.png,Lung_Opacity/images/Lung_Opacity-2410.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2410.png,Lung_Opacity/masks/Lung_Opacity-2410.png,True,False,299,299,L,131.11,199ed62c080e7262e72cd3acfcc7cb93
Lung_Opacity-2411.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2411.png,Lung_Opacity/images/Lung_Opacity-2411.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2411.png,Lung_Opacity/masks/Lung_Opacity-2411.png,True,False,299,299,L,140.47,901a1b9461944a746b6e1ca4033fc247
Lung_Opacity-2412.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2412.png,Lung_Opacity/images/Lung_Opacity-2412.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2412.png,Lung_Opacity/masks/Lung_Opacity-2412.png,True,False,299,299,L,164.08,ba0dab1e7844b43fd1a2b790d3c6e83d
Lung_Opacity-2413.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2413.png,Lung_Opacity/images/Lung_Opacity-2413.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2413.png,Lung_Opacity/masks/Lung_Opacity-2413.png,True,False,299,299,L,139.45,38545160d3d6bc0559bca9a8221e4a19
Lung_Opacity-2414.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2414.png,Lung_Opacity/images/Lung_Opacity-2414.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2414.png,Lung_Opacity/masks/Lung_Opacity-2414.png,True,False,299,299,L,190.16,5fc0288664448259a24671503fffcba3
Lung_Opacity-2415.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2415.png,Lung_Opacity/images/Lung_Opacity-2415.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2415.png,Lung_Opacity/masks/Lung_Opacity-2415.png,True,False,299,299,L,120.15,07f1e72a34fe56bc6dd362f3bd4e10f4
Lung_Opacity-2416.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2416.png,Lung_Opacity/images/Lung_Opacity-2416.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2416.png,Lung_Opacity/masks/Lung_Opacity-2416.png,True,False,299,299,L,99.28,c58fd729b040610b53fe707192ab115b
Lung_Opacity-2417.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2417.png,Lung_Opacity/images/Lung_Opacity-2417.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2417.png,Lung_Opacity/masks/Lung_Opacity-2417.png,True,False,299,299,L,157.7,d0594a567900b6aa964748c6a99d1c60
Lung_Opacity-2418.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2418.png,Lung_Opacity/images/Lung_Opacity-2418.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2418.png,Lung_Opacity/masks/Lung_Opacity-2418.png,True,False,299,299,L,108.78,47eaf7a09124b601ae15ba1bf793766b
Lung_Opacity-2419.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2419.png,Lung_Opacity/images/Lung_Opacity-2419.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2419.png,Lung_Opacity/masks/Lung_Opacity-2419.png,True,False,299,299,L,111.74,c12b69c314ad18b6c8fb7b2865830915
Lung_Opacity-242.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-242.png,Lung_Opacity/images/Lung_Opacity-242.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-242.png,Lung_Opacity/masks/Lung_Opacity-242.png,True,False,299,299,L,134.63,e26077d265fb5faecbee0bd06abed0f1
Lung_Opacity-2420.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2420.png,Lung_Opacity/images/Lung_Opacity-2420.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2420.png,Lung_Opacity/masks/Lung_Opacity-2420.png,True,False,299,299,L,116.72,a04113559bd35cec6f9c78c860c1fc6e
Lung_Opacity-2421.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2421.png,Lung_Opacity/images/Lung_Opacity-2421.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2421.png,Lung_Opacity/masks/Lung_Opacity-2421.png,True,False,299,299,L,105.95,12e59adbf81f9e0e62c2a670d5d2fd05
Lung_Opacity-2422.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2422.png,Lung_Opacity/images/Lung_Opacity-2422.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2422.png,Lung_Opacity/masks/Lung_Opacity-2422.png,True,False,299,299,L,79.44,3bece55be736d98fe338812c1303d6cd
Lung_Opacity-2423.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2423.png,Lung_Opacity/images/Lung_Opacity-2423.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2423.png,Lung_Opacity/masks/Lung_Opacity-2423.png,True,False,299,299,L,79.06,a20e8b0a95fbefb2015082017c1f6a86
Lung_Opacity-2424.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2424.png,Lung_Opacity/images/Lung_Opacity-2424.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2424.png,Lung_Opacity/masks/Lung_Opacity-2424.png,True,False,299,299,L,156.77,5137df3d341f30d04fd23a27ee2ca598
Lung_Opacity-2425.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2425.png,Lung_Opacity/images/Lung_Opacity-2425.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2425.png,Lung_Opacity/masks/Lung_Opacity-2425.png,True,False,299,299,L,116.11,aacf1d1e2f065ff2788e4b1bcc391cb8
Lung_Opacity-2426.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2426.png,Lung_Opacity/images/Lung_Opacity-2426.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2426.png,Lung_Opacity/masks/Lung_Opacity-2426.png,True,False,299,299,L,124.01,54bb58684e0d18026393b950b18ffe8c
Lung_Opacity-2427.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2427.png,Lung_Opacity/images/Lung_Opacity-2427.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2427.png,Lung_Opacity/masks/Lung_Opacity-2427.png,True,False,299,299,L,108.89,b9d05c93616a9e4ad8bd84139b3f6bcb
Lung_Opacity-2428.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2428.png,Lung_Opacity/images/Lung_Opacity-2428.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2428.png,Lung_Opacity/masks/Lung_Opacity-2428.png,True,False,299,299,L,103.86,ca2ee4096e64130c45a8d33646ed962e
Lung_Opacity-2429.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2429.png,Lung_Opacity/images/Lung_Opacity-2429.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2429.png,Lung_Opacity/masks/Lung_Opacity-2429.png,True,False,299,299,L,151.93,41ac91f8c4d61ccca9d10b0416ea4683
Lung_Opacity-243.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-243.png,Lung_Opacity/images/Lung_Opacity-243.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-243.png,Lung_Opacity/masks/Lung_Opacity-243.png,True,False,299,299,L,111.25,bd78ff8e7fab0690fc428f93d63f9c8f
Lung_Opacity-2430.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2430.png,Lung_Opacity/images/Lung_Opacity-2430.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2430.png,Lung_Opacity/masks/Lung_Opacity-2430.png,True,False,299,299,L,122.32,7cf2f642abb435de1523fb61ad6c54ad
Lung_Opacity-2431.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2431.png,Lung_Opacity/images/Lung_Opacity-2431.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2431.png,Lung_Opacity/masks/Lung_Opacity-2431.png,True,False,299,299,L,108.37,70753fa8de3ddb48839d71f9139bf11f
Lung_Opacity-2432.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2432.png,Lung_Opacity/images/Lung_Opacity-2432.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2432.png,Lung_Opacity/masks/Lung_Opacity-2432.png,True,False,299,299,L,156.57,fc2685f81b489a14bd8957c9a1b9665f
Lung_Opacity-2433.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2433.png,Lung_Opacity/images/Lung_Opacity-2433.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2433.png,Lung_Opacity/masks/Lung_Opacity-2433.png,True,False,299,299,L,166.25,41f240fab1bb808f1ffa3608f06e34d4
Lung_Opacity-2434.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2434.png,Lung_Opacity/images/Lung_Opacity-2434.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2434.png,Lung_Opacity/masks/Lung_Opacity-2434.png,True,False,299,299,L,158.19,3619900f067e458ebe336322ad4ad859
Lung_Opacity-2435.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2435.png,Lung_Opacity/images/Lung_Opacity-2435.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2435.png,Lung_Opacity/masks/Lung_Opacity-2435.png,True,False,299,299,L,107.03,9f40b693ffa5f1f76f5418f43dc3afe4
Lung_Opacity-2436.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2436.png,Lung_Opacity/images/Lung_Opacity-2436.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2436.png,Lung_Opacity/masks/Lung_Opacity-2436.png,True,False,299,299,L,166.4,febf4345a30a74988760e99bc6da23bc
Lung_Opacity-2437.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2437.png,Lung_Opacity/images/Lung_Opacity-2437.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2437.png,Lung_Opacity/masks/Lung_Opacity-2437.png,True,False,299,299,L,124.32,375a98f2729862203e46b7103277b4ff
Lung_Opacity-2438.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2438.png,Lung_Opacity/images/Lung_Opacity-2438.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2438.png,Lung_Opacity/masks/Lung_Opacity-2438.png,True,False,299,299,L,120.02,16dfbb111c9a7d59ee5740eb79c2c2c6
Lung_Opacity-2439.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2439.png,Lung_Opacity/images/Lung_Opacity-2439.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2439.png,Lung_Opacity/masks/Lung_Opacity-2439.png,True,False,299,299,L,116.56,3b28fa2bc2701e3ff025537d76177fde
Lung_Opacity-244.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-244.png,Lung_Opacity/images/Lung_Opacity-244.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-244.png,Lung_Opacity/masks/Lung_Opacity-244.png,True,False,299,299,L,127.7,0b6138dfc0ba3c0064a1bd74eb26a832
Lung_Opacity-2440.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2440.png,Lung_Opacity/images/Lung_Opacity-2440.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2440.png,Lung_Opacity/masks/Lung_Opacity-2440.png,True,False,299,299,L,123.99,f7a7d467d8986c0b4edd8fcf2f9cdc34
Lung_Opacity-2441.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2441.png,Lung_Opacity/images/Lung_Opacity-2441.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2441.png,Lung_Opacity/masks/Lung_Opacity-2441.png,True,False,299,299,L,170.47,19c90fb04543291d9ed69eecfe4ccecc
Lung_Opacity-2442.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2442.png,Lung_Opacity/images/Lung_Opacity-2442.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2442.png,Lung_Opacity/masks/Lung_Opacity-2442.png,True,False,299,299,L,113.21,695c6853f5b096384636a2277c63e6c6
Lung_Opacity-2443.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2443.png,Lung_Opacity/images/Lung_Opacity-2443.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2443.png,Lung_Opacity/masks/Lung_Opacity-2443.png,True,False,299,299,L,95.5,51889ae2105bbc2284fae04adfd9f46e
Lung_Opacity-2444.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2444.png,Lung_Opacity/images/Lung_Opacity-2444.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2444.png,Lung_Opacity/masks/Lung_Opacity-2444.png,True,False,299,299,L,120.82,207b6c5c5ea1bf708e8972e9f4dfafec
Lung_Opacity-2445.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2445.png,Lung_Opacity/images/Lung_Opacity-2445.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2445.png,Lung_Opacity/masks/Lung_Opacity-2445.png,True,False,299,299,L,135.72,18637cfcd07fdc4eb379d95946640c91
Lung_Opacity-2446.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2446.png,Lung_Opacity/images/Lung_Opacity-2446.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2446.png,Lung_Opacity/masks/Lung_Opacity-2446.png,True,False,299,299,L,114.69,48570c39f371a1f333f4489aeddfafca
Lung_Opacity-2447.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2447.png,Lung_Opacity/images/Lung_Opacity-2447.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2447.png,Lung_Opacity/masks/Lung_Opacity-2447.png,True,False,299,299,L,105.02,3ad97239e56a62b909fae29ab12eebdf
Lung_Opacity-2448.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2448.png,Lung_Opacity/images/Lung_Opacity-2448.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2448.png,Lung_Opacity/masks/Lung_Opacity-2448.png,True,False,299,299,L,135.89,3d4aec03ca34dcf46011b9a02491d860
Lung_Opacity-2449.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2449.png,Lung_Opacity/images/Lung_Opacity-2449.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2449.png,Lung_Opacity/masks/Lung_Opacity-2449.png,True,False,299,299,L,112.71,18aebad6d3a11369d80f63a24ba9d62b
Lung_Opacity-245.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-245.png,Lung_Opacity/images/Lung_Opacity-245.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-245.png,Lung_Opacity/masks/Lung_Opacity-245.png,True,False,299,299,L,130.41,567610637919b02966178459c219b55f
Lung_Opacity-2450.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2450.png,Lung_Opacity/images/Lung_Opacity-2450.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2450.png,Lung_Opacity/masks/Lung_Opacity-2450.png,True,False,299,299,L,45.23,bc05cc333940b7c66635cd35ec92330f
Lung_Opacity-2451.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2451.png,Lung_Opacity/images/Lung_Opacity-2451.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2451.png,Lung_Opacity/masks/Lung_Opacity-2451.png,True,False,299,299,L,113.2,b69a676c0a6c854c4065a90d636bd689
Lung_Opacity-2452.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2452.png,Lung_Opacity/images/Lung_Opacity-2452.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2452.png,Lung_Opacity/masks/Lung_Opacity-2452.png,True,False,299,299,L,145.32,667cd7109d854b39cd3d468d3dc17d14
Lung_Opacity-2453.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2453.png,Lung_Opacity/images/Lung_Opacity-2453.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2453.png,Lung_Opacity/masks/Lung_Opacity-2453.png,True,False,299,299,L,126.84,7d86663fc81d255485f19091364e1879
Lung_Opacity-2454.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2454.png,Lung_Opacity/images/Lung_Opacity-2454.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2454.png,Lung_Opacity/masks/Lung_Opacity-2454.png,True,False,299,299,L,104.52,bc8c6792f112b3ab1e7fc3e97cc1d978
Lung_Opacity-2455.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2455.png,Lung_Opacity/images/Lung_Opacity-2455.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2455.png,Lung_Opacity/masks/Lung_Opacity-2455.png,True,False,299,299,L,123.33,101d6d5686f527776fdb580538262ab8
Lung_Opacity-2456.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2456.png,Lung_Opacity/images/Lung_Opacity-2456.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2456.png,Lung_Opacity/masks/Lung_Opacity-2456.png,True,False,299,299,L,111.35,b72b0f4223833f54e9b114f8110b3c4d
Lung_Opacity-2457.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2457.png,Lung_Opacity/images/Lung_Opacity-2457.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2457.png,Lung_Opacity/masks/Lung_Opacity-2457.png,True,False,299,299,L,127.84,edbdd7bbaa4699980cc646f0d67ff7d6
Lung_Opacity-2458.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2458.png,Lung_Opacity/images/Lung_Opacity-2458.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2458.png,Lung_Opacity/masks/Lung_Opacity-2458.png,True,False,299,299,L,65.26,ca68952b174e542377081cfa2e0e714d
Lung_Opacity-2459.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2459.png,Lung_Opacity/images/Lung_Opacity-2459.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2459.png,Lung_Opacity/masks/Lung_Opacity-2459.png,True,False,299,299,L,102.55,249f0f8b17fc408e6234baf35e862569
Lung_Opacity-246.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-246.png,Lung_Opacity/images/Lung_Opacity-246.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-246.png,Lung_Opacity/masks/Lung_Opacity-246.png,True,False,299,299,L,133.31,96098e9c0a0b6bfda5b25675d02000ab
Lung_Opacity-2460.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2460.png,Lung_Opacity/images/Lung_Opacity-2460.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2460.png,Lung_Opacity/masks/Lung_Opacity-2460.png,True,False,299,299,L,102.61,8fa850491ecc3863914fd643665702e7
Lung_Opacity-2461.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2461.png,Lung_Opacity/images/Lung_Opacity-2461.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2461.png,Lung_Opacity/masks/Lung_Opacity-2461.png,True,False,299,299,L,122.58,6d71b8c0acc256222c22a29c93862efa
Lung_Opacity-2462.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2462.png,Lung_Opacity/images/Lung_Opacity-2462.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2462.png,Lung_Opacity/masks/Lung_Opacity-2462.png,True,False,299,299,L,152.82,17c42f170ffdc1552711e31bab555832
Lung_Opacity-2463.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2463.png,Lung_Opacity/images/Lung_Opacity-2463.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2463.png,Lung_Opacity/masks/Lung_Opacity-2463.png,True,False,299,299,L,137.44,c3852e76cea5237e953bd3dd29c0294f
Lung_Opacity-2464.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2464.png,Lung_Opacity/images/Lung_Opacity-2464.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2464.png,Lung_Opacity/masks/Lung_Opacity-2464.png,True,False,299,299,L,101.95,60fc6b42499f7826fd918e3bd8359678
Lung_Opacity-2465.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2465.png,Lung_Opacity/images/Lung_Opacity-2465.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2465.png,Lung_Opacity/masks/Lung_Opacity-2465.png,True,False,299,299,L,114.87,3dd8248ba30eb5ffa3f99f6fc7bc238b
Lung_Opacity-2466.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2466.png,Lung_Opacity/images/Lung_Opacity-2466.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2466.png,Lung_Opacity/masks/Lung_Opacity-2466.png,True,False,299,299,L,121.47,07fe11d8240d376aca4d257f02423cee
Lung_Opacity-2467.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2467.png,Lung_Opacity/images/Lung_Opacity-2467.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2467.png,Lung_Opacity/masks/Lung_Opacity-2467.png,True,False,299,299,L,177.95,ee2370b573ff205be754efcb1e235b96
Lung_Opacity-2468.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2468.png,Lung_Opacity/images/Lung_Opacity-2468.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2468.png,Lung_Opacity/masks/Lung_Opacity-2468.png,True,False,299,299,L,111.53,112692841b0c030643a4c75233adf539
Lung_Opacity-2469.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2469.png,Lung_Opacity/images/Lung_Opacity-2469.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2469.png,Lung_Opacity/masks/Lung_Opacity-2469.png,True,False,299,299,L,137.67,047979ddbd5a129bf599bacb451c7660
Lung_Opacity-247.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-247.png,Lung_Opacity/images/Lung_Opacity-247.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-247.png,Lung_Opacity/masks/Lung_Opacity-247.png,True,False,299,299,L,166.19,f4685dc2fc76ae2e7ca18057091811ce
Lung_Opacity-2470.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2470.png,Lung_Opacity/images/Lung_Opacity-2470.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2470.png,Lung_Opacity/masks/Lung_Opacity-2470.png,True,False,299,299,L,141.99,e3e231a170a0372f42ff0064e7ffb0ac
Lung_Opacity-2471.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2471.png,Lung_Opacity/images/Lung_Opacity-2471.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2471.png,Lung_Opacity/masks/Lung_Opacity-2471.png,True,False,299,299,L,103.59,a3f42e6f2e438e68cb9feb45f908df08
Lung_Opacity-2472.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2472.png,Lung_Opacity/images/Lung_Opacity-2472.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2472.png,Lung_Opacity/masks/Lung_Opacity-2472.png,True,False,299,299,L,118.61,051affb8e1235c6cd227565661e22002
Lung_Opacity-2473.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2473.png,Lung_Opacity/images/Lung_Opacity-2473.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2473.png,Lung_Opacity/masks/Lung_Opacity-2473.png,True,False,299,299,L,171.27,84964c7d16c0d8c2e09dd51a69d61e76
Lung_Opacity-2474.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2474.png,Lung_Opacity/images/Lung_Opacity-2474.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2474.png,Lung_Opacity/masks/Lung_Opacity-2474.png,True,False,299,299,L,121.74,af686cec1a14a79ee598ac9919d688da
Lung_Opacity-2475.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2475.png,Lung_Opacity/images/Lung_Opacity-2475.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2475.png,Lung_Opacity/masks/Lung_Opacity-2475.png,True,False,299,299,L,129.51,ea2fd42911685002427788231c77b998
Lung_Opacity-2476.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2476.png,Lung_Opacity/images/Lung_Opacity-2476.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2476.png,Lung_Opacity/masks/Lung_Opacity-2476.png,True,False,299,299,L,100.41,7e1c231a3f761f10145d024303f9bfc4
Lung_Opacity-2477.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2477.png,Lung_Opacity/images/Lung_Opacity-2477.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2477.png,Lung_Opacity/masks/Lung_Opacity-2477.png,True,False,299,299,L,171.04,a8a804d4bef51218d73783721c602979
Lung_Opacity-2478.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2478.png,Lung_Opacity/images/Lung_Opacity-2478.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2478.png,Lung_Opacity/masks/Lung_Opacity-2478.png,True,False,299,299,L,123.23,2608b82083ab02fc47dfa38e5f4903a5
Lung_Opacity-2479.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2479.png,Lung_Opacity/images/Lung_Opacity-2479.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2479.png,Lung_Opacity/masks/Lung_Opacity-2479.png,True,False,299,299,L,120.17,9c94e44b84d5c13113bbe0cfaba6bd61
Lung_Opacity-248.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-248.png,Lung_Opacity/images/Lung_Opacity-248.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-248.png,Lung_Opacity/masks/Lung_Opacity-248.png,True,False,299,299,L,124.37,832d4f28fb60e43d481d9a4d3fdb8416
Lung_Opacity-2480.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2480.png,Lung_Opacity/images/Lung_Opacity-2480.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2480.png,Lung_Opacity/masks/Lung_Opacity-2480.png,True,False,299,299,L,113.25,b67e17edb465401912c028c97923fb37
Lung_Opacity-2481.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2481.png,Lung_Opacity/images/Lung_Opacity-2481.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2481.png,Lung_Opacity/masks/Lung_Opacity-2481.png,True,False,299,299,L,157.68,c5983c98c576d7d3de0afed84a402442
Lung_Opacity-2482.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2482.png,Lung_Opacity/images/Lung_Opacity-2482.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2482.png,Lung_Opacity/masks/Lung_Opacity-2482.png,True,False,299,299,L,132.02,6f414197c76257514a59aaa8e07cf110
Lung_Opacity-2483.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2483.png,Lung_Opacity/images/Lung_Opacity-2483.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2483.png,Lung_Opacity/masks/Lung_Opacity-2483.png,True,False,299,299,L,96.75,d20bcc0bd244ae1e0a73c4b1261452cc
Lung_Opacity-2484.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2484.png,Lung_Opacity/images/Lung_Opacity-2484.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2484.png,Lung_Opacity/masks/Lung_Opacity-2484.png,True,False,299,299,L,157.99,6fa1817b0a3bd08d6d4f1c48e93d1cb0
Lung_Opacity-2485.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2485.png,Lung_Opacity/images/Lung_Opacity-2485.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2485.png,Lung_Opacity/masks/Lung_Opacity-2485.png,True,False,299,299,L,115.89,42da0064d8d97348b80c683d1eddd7fa
Lung_Opacity-2486.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2486.png,Lung_Opacity/images/Lung_Opacity-2486.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2486.png,Lung_Opacity/masks/Lung_Opacity-2486.png,True,False,299,299,L,163.76,a6ee0f66332a81d5c06af11ca4a4274f
Lung_Opacity-2487.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2487.png,Lung_Opacity/images/Lung_Opacity-2487.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2487.png,Lung_Opacity/masks/Lung_Opacity-2487.png,True,False,299,299,L,134.02,9da575441295850f9dcc9bde8bbf5290
Lung_Opacity-2488.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2488.png,Lung_Opacity/images/Lung_Opacity-2488.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2488.png,Lung_Opacity/masks/Lung_Opacity-2488.png,True,False,299,299,L,114.01,b6bb7fa665e0fa30924e219fd1907591
Lung_Opacity-2489.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2489.png,Lung_Opacity/images/Lung_Opacity-2489.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2489.png,Lung_Opacity/masks/Lung_Opacity-2489.png,True,False,299,299,L,73.36,0f5e6e74c92516f8007a673b9f10e3b5
Lung_Opacity-249.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-249.png,Lung_Opacity/images/Lung_Opacity-249.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-249.png,Lung_Opacity/masks/Lung_Opacity-249.png,True,False,299,299,L,147.61,01ef4914f59bb67a4b7a272cca48b055
Lung_Opacity-2490.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2490.png,Lung_Opacity/images/Lung_Opacity-2490.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2490.png,Lung_Opacity/masks/Lung_Opacity-2490.png,True,False,299,299,L,105.55,a298ab7345353c171edd7b0fb4205df6
Lung_Opacity-2491.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2491.png,Lung_Opacity/images/Lung_Opacity-2491.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2491.png,Lung_Opacity/masks/Lung_Opacity-2491.png,True,False,299,299,L,134.4,dfcd70d03f5c742cf491b0e1cf43cad2
Lung_Opacity-2492.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2492.png,Lung_Opacity/images/Lung_Opacity-2492.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2492.png,Lung_Opacity/masks/Lung_Opacity-2492.png,True,False,299,299,L,151.03,69a57c8968d145c0b87b06a3eb2c4214
Lung_Opacity-2493.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2493.png,Lung_Opacity/images/Lung_Opacity-2493.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2493.png,Lung_Opacity/masks/Lung_Opacity-2493.png,True,False,299,299,L,109.64,50482a37315df636e73f4a78acc1f54e
Lung_Opacity-2494.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2494.png,Lung_Opacity/images/Lung_Opacity-2494.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2494.png,Lung_Opacity/masks/Lung_Opacity-2494.png,True,False,299,299,L,172.92,91ed6276c9793c1ae3d383542feef1b2
Lung_Opacity-2495.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2495.png,Lung_Opacity/images/Lung_Opacity-2495.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2495.png,Lung_Opacity/masks/Lung_Opacity-2495.png,True,False,299,299,L,159.48,a6254bc8489077520a672afdf38b8a11
Lung_Opacity-2496.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2496.png,Lung_Opacity/images/Lung_Opacity-2496.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2496.png,Lung_Opacity/masks/Lung_Opacity-2496.png,True,False,299,299,L,118.03,4345e43fb42c782421b7f06dcac609bd
Lung_Opacity-2497.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2497.png,Lung_Opacity/images/Lung_Opacity-2497.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2497.png,Lung_Opacity/masks/Lung_Opacity-2497.png,True,False,299,299,L,135.12,1f5d77cf28e89de9644178dcad132fc8
Lung_Opacity-2498.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2498.png,Lung_Opacity/images/Lung_Opacity-2498.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2498.png,Lung_Opacity/masks/Lung_Opacity-2498.png,True,False,299,299,L,168.89,6bd8dc4b97d9b4aa00bbf9a3d23d41d9
Lung_Opacity-2499.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2499.png,Lung_Opacity/images/Lung_Opacity-2499.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2499.png,Lung_Opacity/masks/Lung_Opacity-2499.png,True,False,299,299,L,160.57,a54c55355d87676fb0c74417cda3672c
Lung_Opacity-25.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-25.png,Lung_Opacity/images/Lung_Opacity-25.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-25.png,Lung_Opacity/masks/Lung_Opacity-25.png,True,False,299,299,L,168.59,5c39f76d7750e55e84cc89d968e51e23
Lung_Opacity-250.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-250.png,Lung_Opacity/images/Lung_Opacity-250.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-250.png,Lung_Opacity/masks/Lung_Opacity-250.png,True,False,299,299,L,100.7,1939f36e718317586c3465faad11ffed
Lung_Opacity-2500.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2500.png,Lung_Opacity/images/Lung_Opacity-2500.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2500.png,Lung_Opacity/masks/Lung_Opacity-2500.png,True,False,299,299,L,161.41,8d06ea553d18fdb2d3e2c9241371dda6
Lung_Opacity-2501.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2501.png,Lung_Opacity/images/Lung_Opacity-2501.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2501.png,Lung_Opacity/masks/Lung_Opacity-2501.png,True,False,299,299,L,117.03,eb096cbe7e545531aa30af15790f7713
Lung_Opacity-2502.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2502.png,Lung_Opacity/images/Lung_Opacity-2502.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2502.png,Lung_Opacity/masks/Lung_Opacity-2502.png,True,False,299,299,L,121.61,4bde91e435a554d172ea8b16015be3e7
Lung_Opacity-2503.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2503.png,Lung_Opacity/images/Lung_Opacity-2503.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2503.png,Lung_Opacity/masks/Lung_Opacity-2503.png,True,False,299,299,L,99.58,e64e916c5dce7c4786404d6871a70c77
Lung_Opacity-2504.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2504.png,Lung_Opacity/images/Lung_Opacity-2504.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2504.png,Lung_Opacity/masks/Lung_Opacity-2504.png,True,False,299,299,L,129.81,ae7cca637778d91eefdbdf45cfbf8436
Lung_Opacity-2505.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2505.png,Lung_Opacity/images/Lung_Opacity-2505.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2505.png,Lung_Opacity/masks/Lung_Opacity-2505.png,True,False,299,299,L,130.26,aa2d96a1a4396cb01591defa97bf6fee
Lung_Opacity-2506.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2506.png,Lung_Opacity/images/Lung_Opacity-2506.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2506.png,Lung_Opacity/masks/Lung_Opacity-2506.png,True,False,299,299,L,134.31,649b6e6ddc087301819e52250f26c987
Lung_Opacity-2507.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2507.png,Lung_Opacity/images/Lung_Opacity-2507.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2507.png,Lung_Opacity/masks/Lung_Opacity-2507.png,True,False,299,299,L,129.27,532ba1029e859bdff7fd872cd2933e33
Lung_Opacity-2508.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2508.png,Lung_Opacity/images/Lung_Opacity-2508.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2508.png,Lung_Opacity/masks/Lung_Opacity-2508.png,True,False,299,299,L,162.93,feb6693d2b38468d88b1c245b96a643f
Lung_Opacity-2509.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2509.png,Lung_Opacity/images/Lung_Opacity-2509.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2509.png,Lung_Opacity/masks/Lung_Opacity-2509.png,True,False,299,299,L,117.3,fa02128f4066b4dd4d873836f2e752cc
Lung_Opacity-251.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-251.png,Lung_Opacity/images/Lung_Opacity-251.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-251.png,Lung_Opacity/masks/Lung_Opacity-251.png,True,False,299,299,L,114.95,af4804e82a188a32dadde5d1e6671527
Lung_Opacity-2510.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2510.png,Lung_Opacity/images/Lung_Opacity-2510.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2510.png,Lung_Opacity/masks/Lung_Opacity-2510.png,True,False,299,299,L,121.85,89518503a4e3c82df09add31adc223f5
Lung_Opacity-2511.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2511.png,Lung_Opacity/images/Lung_Opacity-2511.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2511.png,Lung_Opacity/masks/Lung_Opacity-2511.png,True,False,299,299,L,104.77,c533063c63c17c6e6f05892432b24ee9
Lung_Opacity-2512.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2512.png,Lung_Opacity/images/Lung_Opacity-2512.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2512.png,Lung_Opacity/masks/Lung_Opacity-2512.png,True,False,299,299,L,108.36,ac9d4563d2543c1f61da2b2bd00c3ee2
Lung_Opacity-2513.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2513.png,Lung_Opacity/images/Lung_Opacity-2513.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2513.png,Lung_Opacity/masks/Lung_Opacity-2513.png,True,False,299,299,L,160.59,a67aeec11a83959596d64cbe353e0209
Lung_Opacity-2514.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2514.png,Lung_Opacity/images/Lung_Opacity-2514.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2514.png,Lung_Opacity/masks/Lung_Opacity-2514.png,True,False,299,299,L,135.66,42541af676eb2d0ce764afbf2a01ee14
Lung_Opacity-2515.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2515.png,Lung_Opacity/images/Lung_Opacity-2515.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2515.png,Lung_Opacity/masks/Lung_Opacity-2515.png,True,False,299,299,L,110.45,826c14393c52079dae48fc17981b3eac
Lung_Opacity-2516.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2516.png,Lung_Opacity/images/Lung_Opacity-2516.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2516.png,Lung_Opacity/masks/Lung_Opacity-2516.png,True,False,299,299,L,96.17,422c851f6c621d69b0cbd89693cd22c7
Lung_Opacity-2517.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2517.png,Lung_Opacity/images/Lung_Opacity-2517.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2517.png,Lung_Opacity/masks/Lung_Opacity-2517.png,True,False,299,299,L,55.81,9360af467c13e925bfc168df1a7aa441
Lung_Opacity-2518.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2518.png,Lung_Opacity/images/Lung_Opacity-2518.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2518.png,Lung_Opacity/masks/Lung_Opacity-2518.png,True,False,299,299,L,115.88,f840a6a0a51d83748162289780e241e1
Lung_Opacity-2519.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2519.png,Lung_Opacity/images/Lung_Opacity-2519.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2519.png,Lung_Opacity/masks/Lung_Opacity-2519.png,True,False,299,299,L,143.59,86ebf806a905e979521ebd6b33cc50fd
Lung_Opacity-252.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-252.png,Lung_Opacity/images/Lung_Opacity-252.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-252.png,Lung_Opacity/masks/Lung_Opacity-252.png,True,False,299,299,L,111.95,93af37fdf6fc3bb1cb3bf386f2ba3a16
Lung_Opacity-2520.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2520.png,Lung_Opacity/images/Lung_Opacity-2520.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2520.png,Lung_Opacity/masks/Lung_Opacity-2520.png,True,False,299,299,L,155.01,aa1489a731b1386e79f74b9606267245
Lung_Opacity-2521.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2521.png,Lung_Opacity/images/Lung_Opacity-2521.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2521.png,Lung_Opacity/masks/Lung_Opacity-2521.png,True,False,299,299,L,156.75,944cc6fce5beda6d35ad268420cfd3c0
Lung_Opacity-2522.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2522.png,Lung_Opacity/images/Lung_Opacity-2522.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2522.png,Lung_Opacity/masks/Lung_Opacity-2522.png,True,False,299,299,L,146.4,8ee124a5ac762a7f9f32fe1035244b2b
Lung_Opacity-2523.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2523.png,Lung_Opacity/images/Lung_Opacity-2523.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2523.png,Lung_Opacity/masks/Lung_Opacity-2523.png,True,False,299,299,L,89.31,d39dc2dd06173bab54e7f04545c2a23f
Lung_Opacity-2524.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2524.png,Lung_Opacity/images/Lung_Opacity-2524.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2524.png,Lung_Opacity/masks/Lung_Opacity-2524.png,True,False,299,299,L,73.14,d4efe9ffe7f30a1e6045ddba3cd50446
Lung_Opacity-2525.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2525.png,Lung_Opacity/images/Lung_Opacity-2525.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2525.png,Lung_Opacity/masks/Lung_Opacity-2525.png,True,False,299,299,L,157.8,b32e72212d3ed8e6b4451c75be861f2b
Lung_Opacity-2526.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2526.png,Lung_Opacity/images/Lung_Opacity-2526.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2526.png,Lung_Opacity/masks/Lung_Opacity-2526.png,True,False,299,299,L,118.08,f0e6991a0bcf5556068705033283f69e
Lung_Opacity-2527.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2527.png,Lung_Opacity/images/Lung_Opacity-2527.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2527.png,Lung_Opacity/masks/Lung_Opacity-2527.png,True,False,299,299,L,132.03,30c5832fbc7703f2a9988634bd0f5ed4
Lung_Opacity-2528.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2528.png,Lung_Opacity/images/Lung_Opacity-2528.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2528.png,Lung_Opacity/masks/Lung_Opacity-2528.png,True,False,299,299,L,137.24,012c86d4bd0a3c78559759ebc3acf544
Lung_Opacity-2529.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2529.png,Lung_Opacity/images/Lung_Opacity-2529.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2529.png,Lung_Opacity/masks/Lung_Opacity-2529.png,True,False,299,299,L,133.48,90089f576411e6c4afa9f1b92e7ba53a
Lung_Opacity-253.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-253.png,Lung_Opacity/images/Lung_Opacity-253.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-253.png,Lung_Opacity/masks/Lung_Opacity-253.png,True,False,299,299,L,105.18,3a2da649a2fa8e9e49274650ef53ee42
Lung_Opacity-2530.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2530.png,Lung_Opacity/images/Lung_Opacity-2530.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2530.png,Lung_Opacity/masks/Lung_Opacity-2530.png,True,False,299,299,L,147.01,bbe4cb5e9373e8d5a33780439c2c03ad
Lung_Opacity-2531.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2531.png,Lung_Opacity/images/Lung_Opacity-2531.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2531.png,Lung_Opacity/masks/Lung_Opacity-2531.png,True,False,299,299,L,111.97,fcf65de55989406d13162345ce61574f
Lung_Opacity-2532.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2532.png,Lung_Opacity/images/Lung_Opacity-2532.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2532.png,Lung_Opacity/masks/Lung_Opacity-2532.png,True,False,299,299,L,155.6,02bc2edcb658ddff154e815cdada8ef0
Lung_Opacity-2533.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2533.png,Lung_Opacity/images/Lung_Opacity-2533.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2533.png,Lung_Opacity/masks/Lung_Opacity-2533.png,True,False,299,299,L,101.27,d26122448e9c9e1c0433b6b860d566f1
Lung_Opacity-2534.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2534.png,Lung_Opacity/images/Lung_Opacity-2534.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2534.png,Lung_Opacity/masks/Lung_Opacity-2534.png,True,False,299,299,L,157.45,040e1fa528a3b320ee570f520dc128f0
Lung_Opacity-2535.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2535.png,Lung_Opacity/images/Lung_Opacity-2535.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2535.png,Lung_Opacity/masks/Lung_Opacity-2535.png,True,False,299,299,L,118.79,0383665ee6ed126d01215982cb6448b0
Lung_Opacity-2536.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2536.png,Lung_Opacity/images/Lung_Opacity-2536.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2536.png,Lung_Opacity/masks/Lung_Opacity-2536.png,True,False,299,299,L,148.9,beb2d88972674d5990b85b94de2edab7
Lung_Opacity-2537.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2537.png,Lung_Opacity/images/Lung_Opacity-2537.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2537.png,Lung_Opacity/masks/Lung_Opacity-2537.png,True,False,299,299,L,115.39,e27344d6f647b9d5c55a0774c33fac21
Lung_Opacity-2538.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2538.png,Lung_Opacity/images/Lung_Opacity-2538.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2538.png,Lung_Opacity/masks/Lung_Opacity-2538.png,True,False,299,299,L,110.31,dbc75335a173530bdee9552182eccdd4
Lung_Opacity-2539.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2539.png,Lung_Opacity/images/Lung_Opacity-2539.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2539.png,Lung_Opacity/masks/Lung_Opacity-2539.png,True,False,299,299,L,97.96,8ee75756165e727797cc4f7e50d0a7a3
Lung_Opacity-254.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-254.png,Lung_Opacity/images/Lung_Opacity-254.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-254.png,Lung_Opacity/masks/Lung_Opacity-254.png,True,False,299,299,L,120.61,ab4d067dd1efab9b064a297deae3e022
Lung_Opacity-2540.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2540.png,Lung_Opacity/images/Lung_Opacity-2540.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2540.png,Lung_Opacity/masks/Lung_Opacity-2540.png,True,False,299,299,L,133.16,bff7f832d8bbe351fff4c52a05070eb6
Lung_Opacity-2541.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2541.png,Lung_Opacity/images/Lung_Opacity-2541.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2541.png,Lung_Opacity/masks/Lung_Opacity-2541.png,True,False,299,299,L,151.19,346f06d72bfb3a2aac6e037eaf0251f3
Lung_Opacity-2542.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2542.png,Lung_Opacity/images/Lung_Opacity-2542.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2542.png,Lung_Opacity/masks/Lung_Opacity-2542.png,True,False,299,299,L,104.74,c868c197e55506b315722b046cb917a2
Lung_Opacity-2543.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2543.png,Lung_Opacity/images/Lung_Opacity-2543.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2543.png,Lung_Opacity/masks/Lung_Opacity-2543.png,True,False,299,299,L,178.66,16949f433dd5996adfb5ae3231a1b178
Lung_Opacity-2544.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2544.png,Lung_Opacity/images/Lung_Opacity-2544.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2544.png,Lung_Opacity/masks/Lung_Opacity-2544.png,True,False,299,299,L,74.78,890f19cbe1a2e5bc43020e61f993d90c
Lung_Opacity-2545.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2545.png,Lung_Opacity/images/Lung_Opacity-2545.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2545.png,Lung_Opacity/masks/Lung_Opacity-2545.png,True,False,299,299,L,128.25,ff7adf0f333ead697d2b9aa58b35a2e4
Lung_Opacity-2546.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2546.png,Lung_Opacity/images/Lung_Opacity-2546.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2546.png,Lung_Opacity/masks/Lung_Opacity-2546.png,True,False,299,299,L,64.15,c0b152c3bc93fb8fd0359b6f4a4b8308
Lung_Opacity-2547.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2547.png,Lung_Opacity/images/Lung_Opacity-2547.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2547.png,Lung_Opacity/masks/Lung_Opacity-2547.png,True,False,299,299,L,138.99,fab0463bdef85f3dc08b92a61ecb0291
Lung_Opacity-2548.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2548.png,Lung_Opacity/images/Lung_Opacity-2548.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2548.png,Lung_Opacity/masks/Lung_Opacity-2548.png,True,False,299,299,L,132.62,b1e27e9870f602edde2c204612c82f83
Lung_Opacity-2549.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2549.png,Lung_Opacity/images/Lung_Opacity-2549.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2549.png,Lung_Opacity/masks/Lung_Opacity-2549.png,True,False,299,299,L,130.13,b94dff4b974e63f9647b15bf6d5ecb9d
Lung_Opacity-255.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-255.png,Lung_Opacity/images/Lung_Opacity-255.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-255.png,Lung_Opacity/masks/Lung_Opacity-255.png,True,False,299,299,L,127.71,3255b6ebe9b305f200470420859897ea
Lung_Opacity-2550.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2550.png,Lung_Opacity/images/Lung_Opacity-2550.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2550.png,Lung_Opacity/masks/Lung_Opacity-2550.png,True,False,299,299,L,130.2,623efa44105c4f9bfbcde385a6639642
Lung_Opacity-2551.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2551.png,Lung_Opacity/images/Lung_Opacity-2551.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2551.png,Lung_Opacity/masks/Lung_Opacity-2551.png,True,False,299,299,L,120.75,97cac1124d9895d1e05180e89af39b40
Lung_Opacity-2552.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2552.png,Lung_Opacity/images/Lung_Opacity-2552.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2552.png,Lung_Opacity/masks/Lung_Opacity-2552.png,True,False,299,299,L,104.97,f66da7aba7bbcc57a8069317b30934d8
Lung_Opacity-2553.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2553.png,Lung_Opacity/images/Lung_Opacity-2553.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2553.png,Lung_Opacity/masks/Lung_Opacity-2553.png,True,False,299,299,L,111.13,bb04c8cda3daf8a20e364389858f978a
Lung_Opacity-2554.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2554.png,Lung_Opacity/images/Lung_Opacity-2554.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2554.png,Lung_Opacity/masks/Lung_Opacity-2554.png,True,False,299,299,L,122.56,d1d66f1ebbe20f3a5487c5db82605786
Lung_Opacity-2555.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2555.png,Lung_Opacity/images/Lung_Opacity-2555.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2555.png,Lung_Opacity/masks/Lung_Opacity-2555.png,True,False,299,299,L,173.11,8458d43e60f8cfaa1c5ed3273895d3a1
Lung_Opacity-2556.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2556.png,Lung_Opacity/images/Lung_Opacity-2556.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2556.png,Lung_Opacity/masks/Lung_Opacity-2556.png,True,False,299,299,L,126.07,294baf06a024e4b170b8e0ee1b4d1d4e
Lung_Opacity-2557.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2557.png,Lung_Opacity/images/Lung_Opacity-2557.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2557.png,Lung_Opacity/masks/Lung_Opacity-2557.png,True,False,299,299,L,111.35,7d012859163080852dba43c402405287
Lung_Opacity-2558.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2558.png,Lung_Opacity/images/Lung_Opacity-2558.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2558.png,Lung_Opacity/masks/Lung_Opacity-2558.png,True,False,299,299,L,140.7,1869e9695d5233a60d5c141c4c7e23bb
Lung_Opacity-2559.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2559.png,Lung_Opacity/images/Lung_Opacity-2559.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2559.png,Lung_Opacity/masks/Lung_Opacity-2559.png,True,False,299,299,L,108.01,521fc16f653a885b1f6864e9ad6defbf
Lung_Opacity-256.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-256.png,Lung_Opacity/images/Lung_Opacity-256.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-256.png,Lung_Opacity/masks/Lung_Opacity-256.png,True,False,299,299,L,77.5,8cf714f0022ca06b728d9f0184feaee1
Lung_Opacity-2560.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2560.png,Lung_Opacity/images/Lung_Opacity-2560.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2560.png,Lung_Opacity/masks/Lung_Opacity-2560.png,True,False,299,299,L,146.24,d691b9ff4f5f9f3f5e76727a08e2568a
Lung_Opacity-2561.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2561.png,Lung_Opacity/images/Lung_Opacity-2561.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2561.png,Lung_Opacity/masks/Lung_Opacity-2561.png,True,False,299,299,L,147.51,1c4de6b87d5c68fb651b04e8d2bc6b50
Lung_Opacity-2562.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2562.png,Lung_Opacity/images/Lung_Opacity-2562.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2562.png,Lung_Opacity/masks/Lung_Opacity-2562.png,True,False,299,299,L,132.75,84914a1b290a719a6198de32ad98f8b3
Lung_Opacity-2563.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2563.png,Lung_Opacity/images/Lung_Opacity-2563.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2563.png,Lung_Opacity/masks/Lung_Opacity-2563.png,True,False,299,299,L,150.57,95f2bb23226c7cb808456c920972cd48
Lung_Opacity-2564.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2564.png,Lung_Opacity/images/Lung_Opacity-2564.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2564.png,Lung_Opacity/masks/Lung_Opacity-2564.png,True,False,299,299,L,94.99,087af5e5cd5dd622e0ef14b2047e9f8e
Lung_Opacity-2565.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2565.png,Lung_Opacity/images/Lung_Opacity-2565.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2565.png,Lung_Opacity/masks/Lung_Opacity-2565.png,True,False,299,299,L,107.25,d0323b9c8d7a1f882afb646f30814327
Lung_Opacity-2566.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2566.png,Lung_Opacity/images/Lung_Opacity-2566.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2566.png,Lung_Opacity/masks/Lung_Opacity-2566.png,True,False,299,299,L,129.73,ae5735414de150c52ba07b91860e8027
Lung_Opacity-2567.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2567.png,Lung_Opacity/images/Lung_Opacity-2567.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2567.png,Lung_Opacity/masks/Lung_Opacity-2567.png,True,False,299,299,L,110.26,e1e9da70baaa83f5a3346479a7dec18b
Lung_Opacity-2568.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2568.png,Lung_Opacity/images/Lung_Opacity-2568.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2568.png,Lung_Opacity/masks/Lung_Opacity-2568.png,True,False,299,299,L,107.33,77ab0dafa444bb289964ccf97dfda3ec
Lung_Opacity-2569.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2569.png,Lung_Opacity/images/Lung_Opacity-2569.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2569.png,Lung_Opacity/masks/Lung_Opacity-2569.png,True,False,299,299,L,77.64,8abab779927cf713af17a24e051149fc
Lung_Opacity-257.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-257.png,Lung_Opacity/images/Lung_Opacity-257.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-257.png,Lung_Opacity/masks/Lung_Opacity-257.png,True,False,299,299,L,107.66,de236ecb334d70d255cc71fcd1df3327
Lung_Opacity-2570.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2570.png,Lung_Opacity/images/Lung_Opacity-2570.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2570.png,Lung_Opacity/masks/Lung_Opacity-2570.png,True,False,299,299,L,162.99,70e4a2b651116c9081540a41674d6498
Lung_Opacity-2571.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2571.png,Lung_Opacity/images/Lung_Opacity-2571.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2571.png,Lung_Opacity/masks/Lung_Opacity-2571.png,True,False,299,299,L,141.25,03763312df19450b880f9e0644ecc70e
Lung_Opacity-2572.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2572.png,Lung_Opacity/images/Lung_Opacity-2572.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2572.png,Lung_Opacity/masks/Lung_Opacity-2572.png,True,False,299,299,L,112.19,b3e1a1b5a3cc6899b25896ae295d2c5f
Lung_Opacity-2573.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2573.png,Lung_Opacity/images/Lung_Opacity-2573.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2573.png,Lung_Opacity/masks/Lung_Opacity-2573.png,True,False,299,299,L,115.34,e6134610de6c42d004e516e3dcd91fec
Lung_Opacity-2574.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2574.png,Lung_Opacity/images/Lung_Opacity-2574.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2574.png,Lung_Opacity/masks/Lung_Opacity-2574.png,True,False,299,299,L,112.34,326cb78d46ad44d2ae24aaf878f17f86
Lung_Opacity-2575.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2575.png,Lung_Opacity/images/Lung_Opacity-2575.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2575.png,Lung_Opacity/masks/Lung_Opacity-2575.png,True,False,299,299,L,104.19,b2921563fc06620543d45056469a89c2
Lung_Opacity-2576.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2576.png,Lung_Opacity/images/Lung_Opacity-2576.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2576.png,Lung_Opacity/masks/Lung_Opacity-2576.png,True,False,299,299,L,109.72,ae45cf24834e2d2a8c4f1125b37f72ac
Lung_Opacity-2577.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2577.png,Lung_Opacity/images/Lung_Opacity-2577.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2577.png,Lung_Opacity/masks/Lung_Opacity-2577.png,True,False,299,299,L,120.14,4016b6393ed550f8a93087a996fad442
Lung_Opacity-2578.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2578.png,Lung_Opacity/images/Lung_Opacity-2578.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2578.png,Lung_Opacity/masks/Lung_Opacity-2578.png,True,False,299,299,L,114.39,d8a012d8e252b3b8a115edab3d8c9e4e
Lung_Opacity-2579.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2579.png,Lung_Opacity/images/Lung_Opacity-2579.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2579.png,Lung_Opacity/masks/Lung_Opacity-2579.png,True,False,299,299,L,121.27,fbedc0acbdcd8de70a78d3afd620c2ce
Lung_Opacity-258.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-258.png,Lung_Opacity/images/Lung_Opacity-258.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-258.png,Lung_Opacity/masks/Lung_Opacity-258.png,True,False,299,299,L,169.67,2e86d09e66a01f5c73736e83ea5ee501
Lung_Opacity-2580.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2580.png,Lung_Opacity/images/Lung_Opacity-2580.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2580.png,Lung_Opacity/masks/Lung_Opacity-2580.png,True,False,299,299,L,150.46,1a08db780263615a7424abbb1fee2775
Lung_Opacity-2581.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2581.png,Lung_Opacity/images/Lung_Opacity-2581.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2581.png,Lung_Opacity/masks/Lung_Opacity-2581.png,True,False,299,299,L,102.07,933e2850488307023a6811075ac753a5
Lung_Opacity-2582.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2582.png,Lung_Opacity/images/Lung_Opacity-2582.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2582.png,Lung_Opacity/masks/Lung_Opacity-2582.png,True,False,299,299,L,112.3,ac58642d7a16d3a88a7b7dfa66d2b156
Lung_Opacity-2583.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2583.png,Lung_Opacity/images/Lung_Opacity-2583.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2583.png,Lung_Opacity/masks/Lung_Opacity-2583.png,True,False,299,299,L,131.2,a6e2cc2805323a36d176314689214d01
Lung_Opacity-2584.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2584.png,Lung_Opacity/images/Lung_Opacity-2584.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2584.png,Lung_Opacity/masks/Lung_Opacity-2584.png,True,False,299,299,L,97.67,4075acc317a60f3bd2c5461ee9c149e5
Lung_Opacity-2585.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2585.png,Lung_Opacity/images/Lung_Opacity-2585.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2585.png,Lung_Opacity/masks/Lung_Opacity-2585.png,True,False,299,299,L,154.67,afbea58494feaf20ae87f6b9f93379b1
Lung_Opacity-2586.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2586.png,Lung_Opacity/images/Lung_Opacity-2586.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2586.png,Lung_Opacity/masks/Lung_Opacity-2586.png,True,False,299,299,L,106.39,b29c1e8058da483cfb771870c37384d5
Lung_Opacity-2587.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2587.png,Lung_Opacity/images/Lung_Opacity-2587.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2587.png,Lung_Opacity/masks/Lung_Opacity-2587.png,True,False,299,299,L,108.87,8b7671da91997b42bab419ae41e0c901
Lung_Opacity-2588.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2588.png,Lung_Opacity/images/Lung_Opacity-2588.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2588.png,Lung_Opacity/masks/Lung_Opacity-2588.png,True,False,299,299,L,109.78,77e81f439994c3c1a677a029e464f1ed
Lung_Opacity-2589.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2589.png,Lung_Opacity/images/Lung_Opacity-2589.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2589.png,Lung_Opacity/masks/Lung_Opacity-2589.png,True,False,299,299,L,101.95,6ff1523f3a5d79f6016482d45bbbb9f5
Lung_Opacity-259.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-259.png,Lung_Opacity/images/Lung_Opacity-259.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-259.png,Lung_Opacity/masks/Lung_Opacity-259.png,True,False,299,299,L,131.98,4a3995be79547d4b1558fc54f164b8d6
Lung_Opacity-2590.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2590.png,Lung_Opacity/images/Lung_Opacity-2590.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2590.png,Lung_Opacity/masks/Lung_Opacity-2590.png,True,False,299,299,L,110.48,00d7317f5c74d3dfce40bfa0db001e63
Lung_Opacity-2591.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2591.png,Lung_Opacity/images/Lung_Opacity-2591.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2591.png,Lung_Opacity/masks/Lung_Opacity-2591.png,True,False,299,299,L,78.71,5352e41b3141d28ec7e3d449fbf80065
Lung_Opacity-2592.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2592.png,Lung_Opacity/images/Lung_Opacity-2592.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2592.png,Lung_Opacity/masks/Lung_Opacity-2592.png,True,False,299,299,L,138.84,a5ae9d914d6d17dfe1a031ddd4ed73dc
Lung_Opacity-2593.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2593.png,Lung_Opacity/images/Lung_Opacity-2593.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2593.png,Lung_Opacity/masks/Lung_Opacity-2593.png,True,False,299,299,L,109.37,ce2ae9ed5778fbac9188cde950e7a95c
Lung_Opacity-2594.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2594.png,Lung_Opacity/images/Lung_Opacity-2594.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2594.png,Lung_Opacity/masks/Lung_Opacity-2594.png,True,False,299,299,L,129.63,e758ca567423cbb1a15c871529d0495b
Lung_Opacity-2595.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2595.png,Lung_Opacity/images/Lung_Opacity-2595.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2595.png,Lung_Opacity/masks/Lung_Opacity-2595.png,True,False,299,299,L,109.84,266f28c9b0621899ea9d9d71df7745c8
Lung_Opacity-2596.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2596.png,Lung_Opacity/images/Lung_Opacity-2596.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2596.png,Lung_Opacity/masks/Lung_Opacity-2596.png,True,False,299,299,L,109.1,4c20843f4f0176a9873f16fe354898e1
Lung_Opacity-2597.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2597.png,Lung_Opacity/images/Lung_Opacity-2597.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2597.png,Lung_Opacity/masks/Lung_Opacity-2597.png,True,False,299,299,L,115.08,66ce29737bc7732d6b6f4131695167c2
Lung_Opacity-2598.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2598.png,Lung_Opacity/images/Lung_Opacity-2598.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2598.png,Lung_Opacity/masks/Lung_Opacity-2598.png,True,False,299,299,L,154.47,28c5612de3b119a8b056bd7d92893e37
Lung_Opacity-2599.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2599.png,Lung_Opacity/images/Lung_Opacity-2599.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2599.png,Lung_Opacity/masks/Lung_Opacity-2599.png,True,False,299,299,L,164.25,ca2b318dc5d9486d03060824802b709c
Lung_Opacity-26.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-26.png,Lung_Opacity/images/Lung_Opacity-26.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-26.png,Lung_Opacity/masks/Lung_Opacity-26.png,True,False,299,299,L,138.87,b4958db92d1e2d098ae7270e2c468240
Lung_Opacity-260.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-260.png,Lung_Opacity/images/Lung_Opacity-260.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-260.png,Lung_Opacity/masks/Lung_Opacity-260.png,True,False,299,299,L,114.84,e1b624e2c413160fc4c494c3c7c5d69a
Lung_Opacity-2600.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2600.png,Lung_Opacity/images/Lung_Opacity-2600.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2600.png,Lung_Opacity/masks/Lung_Opacity-2600.png,True,False,299,299,L,107.37,c722fe7b6731d849d7e4fdff806dadbd
Lung_Opacity-2601.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2601.png,Lung_Opacity/images/Lung_Opacity-2601.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2601.png,Lung_Opacity/masks/Lung_Opacity-2601.png,True,False,299,299,L,110.0,ead668bc5bee38a23573ce972a239e91
Lung_Opacity-2602.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2602.png,Lung_Opacity/images/Lung_Opacity-2602.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2602.png,Lung_Opacity/masks/Lung_Opacity-2602.png,True,False,299,299,L,115.06,c7d4015af17f99b6bc0a19a3ed4ccc90
Lung_Opacity-2603.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2603.png,Lung_Opacity/images/Lung_Opacity-2603.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2603.png,Lung_Opacity/masks/Lung_Opacity-2603.png,True,False,299,299,L,101.59,2f7df87231e49ad64aa7c2603543d52b
Lung_Opacity-2604.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2604.png,Lung_Opacity/images/Lung_Opacity-2604.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2604.png,Lung_Opacity/masks/Lung_Opacity-2604.png,True,False,299,299,L,81.76,706520d1e9ee1d561186dfb04043cccb
Lung_Opacity-2605.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2605.png,Lung_Opacity/images/Lung_Opacity-2605.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2605.png,Lung_Opacity/masks/Lung_Opacity-2605.png,True,False,299,299,L,134.2,8dec908a449496555c48ea8e003eef87
Lung_Opacity-2606.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2606.png,Lung_Opacity/images/Lung_Opacity-2606.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2606.png,Lung_Opacity/masks/Lung_Opacity-2606.png,True,False,299,299,L,153.3,b8cce1f88672b33b484293253cb9d8d6
Lung_Opacity-2607.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2607.png,Lung_Opacity/images/Lung_Opacity-2607.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2607.png,Lung_Opacity/masks/Lung_Opacity-2607.png,True,False,299,299,L,111.88,36fce5c05f3da5836de54b2a99860ad0
Lung_Opacity-2608.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2608.png,Lung_Opacity/images/Lung_Opacity-2608.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2608.png,Lung_Opacity/masks/Lung_Opacity-2608.png,True,False,299,299,L,171.45,bf89bbbc9f42e4b046b22db468b2bdbe
Lung_Opacity-2609.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2609.png,Lung_Opacity/images/Lung_Opacity-2609.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2609.png,Lung_Opacity/masks/Lung_Opacity-2609.png,True,False,299,299,L,109.71,397bc47b792671e21a92d8b60667a3e4
Lung_Opacity-261.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-261.png,Lung_Opacity/images/Lung_Opacity-261.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-261.png,Lung_Opacity/masks/Lung_Opacity-261.png,True,False,299,299,L,104.17,2d4b13513edfb680b37619fe7e2c4685
Lung_Opacity-2610.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2610.png,Lung_Opacity/images/Lung_Opacity-2610.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2610.png,Lung_Opacity/masks/Lung_Opacity-2610.png,True,False,299,299,L,100.72,eb95a5ed273bd04693e4737f1216ef55
Lung_Opacity-2611.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2611.png,Lung_Opacity/images/Lung_Opacity-2611.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2611.png,Lung_Opacity/masks/Lung_Opacity-2611.png,True,False,299,299,L,149.92,4f003b7b3eebce0795e2113ba1da23e8
Lung_Opacity-2612.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2612.png,Lung_Opacity/images/Lung_Opacity-2612.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2612.png,Lung_Opacity/masks/Lung_Opacity-2612.png,True,False,299,299,L,156.27,c2cdcb5f169acaaa71e96c65aa1c5670
Lung_Opacity-2613.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2613.png,Lung_Opacity/images/Lung_Opacity-2613.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2613.png,Lung_Opacity/masks/Lung_Opacity-2613.png,True,False,299,299,L,122.86,fd879a691bc0f748884438b86c133344
Lung_Opacity-2614.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2614.png,Lung_Opacity/images/Lung_Opacity-2614.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2614.png,Lung_Opacity/masks/Lung_Opacity-2614.png,True,False,299,299,L,100.44,423a789402d328b93e61ffd39213b806
Lung_Opacity-2615.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2615.png,Lung_Opacity/images/Lung_Opacity-2615.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2615.png,Lung_Opacity/masks/Lung_Opacity-2615.png,True,False,299,299,L,110.72,74503bc72bf8c31f663e27ff33cb9eb4
Lung_Opacity-2616.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2616.png,Lung_Opacity/images/Lung_Opacity-2616.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2616.png,Lung_Opacity/masks/Lung_Opacity-2616.png,True,False,299,299,L,105.48,b46ff06fb2b71f341af8a74677b47f6c
Lung_Opacity-2617.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2617.png,Lung_Opacity/images/Lung_Opacity-2617.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2617.png,Lung_Opacity/masks/Lung_Opacity-2617.png,True,False,299,299,L,119.61,95a0a58727134307a2324121b940345c
Lung_Opacity-2618.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2618.png,Lung_Opacity/images/Lung_Opacity-2618.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2618.png,Lung_Opacity/masks/Lung_Opacity-2618.png,True,False,299,299,L,136.47,d2729d6cc97946c6d7063d63144d8497
Lung_Opacity-2619.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2619.png,Lung_Opacity/images/Lung_Opacity-2619.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2619.png,Lung_Opacity/masks/Lung_Opacity-2619.png,True,False,299,299,L,125.71,f77ac80e9f61eff305d37eed3c85d0ba
Lung_Opacity-262.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-262.png,Lung_Opacity/images/Lung_Opacity-262.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-262.png,Lung_Opacity/masks/Lung_Opacity-262.png,True,False,299,299,L,105.74,67736addbddb4c4ff2dda7a573a65130
Lung_Opacity-2620.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2620.png,Lung_Opacity/images/Lung_Opacity-2620.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2620.png,Lung_Opacity/masks/Lung_Opacity-2620.png,True,False,299,299,L,112.81,0ebab7904b47c55408a8e640b896b8f1
Lung_Opacity-2621.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2621.png,Lung_Opacity/images/Lung_Opacity-2621.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2621.png,Lung_Opacity/masks/Lung_Opacity-2621.png,True,False,299,299,L,63.48,12450b7a8498746fac083a786f38b1c8
Lung_Opacity-2622.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2622.png,Lung_Opacity/images/Lung_Opacity-2622.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2622.png,Lung_Opacity/masks/Lung_Opacity-2622.png,True,False,299,299,L,140.54,f955651244b7d6b3eab760ea279a038d
Lung_Opacity-2623.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2623.png,Lung_Opacity/images/Lung_Opacity-2623.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2623.png,Lung_Opacity/masks/Lung_Opacity-2623.png,True,False,299,299,L,149.1,1fe65324836d05ae8263683dcd276a94
Lung_Opacity-2624.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2624.png,Lung_Opacity/images/Lung_Opacity-2624.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2624.png,Lung_Opacity/masks/Lung_Opacity-2624.png,True,False,299,299,L,114.81,fae28d0672479cc447fcff4345f4a741
Lung_Opacity-2625.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2625.png,Lung_Opacity/images/Lung_Opacity-2625.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2625.png,Lung_Opacity/masks/Lung_Opacity-2625.png,True,False,299,299,L,107.79,813346ea10369cdbe8719c2576566470
Lung_Opacity-2626.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2626.png,Lung_Opacity/images/Lung_Opacity-2626.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2626.png,Lung_Opacity/masks/Lung_Opacity-2626.png,True,False,299,299,L,142.44,e1f3bdb2281b53b01294bdbd6fc5619b
Lung_Opacity-2627.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2627.png,Lung_Opacity/images/Lung_Opacity-2627.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2627.png,Lung_Opacity/masks/Lung_Opacity-2627.png,True,False,299,299,L,163.6,d5d7ae24e24268da245859f4c3f1c13e
Lung_Opacity-2628.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2628.png,Lung_Opacity/images/Lung_Opacity-2628.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2628.png,Lung_Opacity/masks/Lung_Opacity-2628.png,True,False,299,299,L,129.84,105c73878d18131887902f4271f3e99a
Lung_Opacity-2629.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2629.png,Lung_Opacity/images/Lung_Opacity-2629.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2629.png,Lung_Opacity/masks/Lung_Opacity-2629.png,True,False,299,299,L,117.85,81e5a40192a52de1c085ddacec4b6bf2
Lung_Opacity-263.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-263.png,Lung_Opacity/images/Lung_Opacity-263.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-263.png,Lung_Opacity/masks/Lung_Opacity-263.png,True,False,299,299,L,159.86,04140767946362a8980b6712d46e1361
Lung_Opacity-2630.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2630.png,Lung_Opacity/images/Lung_Opacity-2630.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2630.png,Lung_Opacity/masks/Lung_Opacity-2630.png,True,False,299,299,L,121.83,85f9747238e2403ddfc328dc9645ad88
Lung_Opacity-2631.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2631.png,Lung_Opacity/images/Lung_Opacity-2631.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2631.png,Lung_Opacity/masks/Lung_Opacity-2631.png,True,False,299,299,L,100.15,7ab56e4b8e188030acb7ba47727f7104
Lung_Opacity-2632.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2632.png,Lung_Opacity/images/Lung_Opacity-2632.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2632.png,Lung_Opacity/masks/Lung_Opacity-2632.png,True,False,299,299,L,177.64,ec039d927b14d4ad7c0d77972328dd78
Lung_Opacity-2633.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2633.png,Lung_Opacity/images/Lung_Opacity-2633.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2633.png,Lung_Opacity/masks/Lung_Opacity-2633.png,True,False,299,299,L,129.87,99f6a84490142259caeecddbfc7cd752
Lung_Opacity-2634.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2634.png,Lung_Opacity/images/Lung_Opacity-2634.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2634.png,Lung_Opacity/masks/Lung_Opacity-2634.png,True,False,299,299,L,122.21,017260758febad5e3d475e21ef68b5e3
Lung_Opacity-2635.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2635.png,Lung_Opacity/images/Lung_Opacity-2635.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2635.png,Lung_Opacity/masks/Lung_Opacity-2635.png,True,False,299,299,L,89.75,7c10829d342ab1223f5a0c01955f9b00
Lung_Opacity-2636.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2636.png,Lung_Opacity/images/Lung_Opacity-2636.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2636.png,Lung_Opacity/masks/Lung_Opacity-2636.png,True,False,299,299,L,123.57,67c9facef93f6076f27cc1294bf9d319
Lung_Opacity-2637.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2637.png,Lung_Opacity/images/Lung_Opacity-2637.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2637.png,Lung_Opacity/masks/Lung_Opacity-2637.png,True,False,299,299,L,92.61,79a33a5bad6d7d682117a9a0dc3b5e18
Lung_Opacity-2638.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2638.png,Lung_Opacity/images/Lung_Opacity-2638.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2638.png,Lung_Opacity/masks/Lung_Opacity-2638.png,True,False,299,299,L,144.43,03f0238afcef5932cb95f46ffe39129d
Lung_Opacity-2639.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2639.png,Lung_Opacity/images/Lung_Opacity-2639.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2639.png,Lung_Opacity/masks/Lung_Opacity-2639.png,True,False,299,299,L,156.39,bf6a1061569c938da554be317f078397
Lung_Opacity-264.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-264.png,Lung_Opacity/images/Lung_Opacity-264.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-264.png,Lung_Opacity/masks/Lung_Opacity-264.png,True,False,299,299,L,134.99,9a2086088d5e5d6eaa63a2eb44440bb5
Lung_Opacity-2640.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2640.png,Lung_Opacity/images/Lung_Opacity-2640.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2640.png,Lung_Opacity/masks/Lung_Opacity-2640.png,True,False,299,299,L,158.43,aa483bc79027b013156069b67a264672
Lung_Opacity-2641.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2641.png,Lung_Opacity/images/Lung_Opacity-2641.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2641.png,Lung_Opacity/masks/Lung_Opacity-2641.png,True,False,299,299,L,139.74,f0c6c43c42a8c9cf2234835dd758820c
Lung_Opacity-2642.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2642.png,Lung_Opacity/images/Lung_Opacity-2642.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2642.png,Lung_Opacity/masks/Lung_Opacity-2642.png,True,False,299,299,L,101.65,926b715f8146ca9157f3bc7638078505
Lung_Opacity-2643.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2643.png,Lung_Opacity/images/Lung_Opacity-2643.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2643.png,Lung_Opacity/masks/Lung_Opacity-2643.png,True,False,299,299,L,139.86,aed8e94a9aa982558e20f1801fa3c41e
Lung_Opacity-2644.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2644.png,Lung_Opacity/images/Lung_Opacity-2644.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2644.png,Lung_Opacity/masks/Lung_Opacity-2644.png,True,False,299,299,L,119.67,306e5e396025c17e85fcc456ca56d911
Lung_Opacity-2645.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2645.png,Lung_Opacity/images/Lung_Opacity-2645.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2645.png,Lung_Opacity/masks/Lung_Opacity-2645.png,True,False,299,299,L,137.34,22aaf90fa49770ada123f6d76b0fd0d9
Lung_Opacity-2646.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2646.png,Lung_Opacity/images/Lung_Opacity-2646.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2646.png,Lung_Opacity/masks/Lung_Opacity-2646.png,True,False,299,299,L,106.21,a65dd84056385e22b9c15bb5fcdc3ed4
Lung_Opacity-2647.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2647.png,Lung_Opacity/images/Lung_Opacity-2647.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2647.png,Lung_Opacity/masks/Lung_Opacity-2647.png,True,False,299,299,L,49.9,cfdfa59e4afb054977bd5ab140d08e94
Lung_Opacity-2648.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2648.png,Lung_Opacity/images/Lung_Opacity-2648.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2648.png,Lung_Opacity/masks/Lung_Opacity-2648.png,True,False,299,299,L,154.62,fc12c5cbfae25db09ac08af883b3965e
Lung_Opacity-2649.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2649.png,Lung_Opacity/images/Lung_Opacity-2649.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2649.png,Lung_Opacity/masks/Lung_Opacity-2649.png,True,False,299,299,L,112.48,ecb99762a338eee6f76342a379e7d274
Lung_Opacity-265.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-265.png,Lung_Opacity/images/Lung_Opacity-265.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-265.png,Lung_Opacity/masks/Lung_Opacity-265.png,True,False,299,299,L,115.68,58b219e24277101383386618cc4712fd
Lung_Opacity-2650.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2650.png,Lung_Opacity/images/Lung_Opacity-2650.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2650.png,Lung_Opacity/masks/Lung_Opacity-2650.png,True,False,299,299,L,111.51,1e39eb5691be708d19ff5b21163643b6
Lung_Opacity-2651.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2651.png,Lung_Opacity/images/Lung_Opacity-2651.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2651.png,Lung_Opacity/masks/Lung_Opacity-2651.png,True,False,299,299,L,132.92,5df72e9ab6f2174b3d6945683d5e7dde
Lung_Opacity-2652.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2652.png,Lung_Opacity/images/Lung_Opacity-2652.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2652.png,Lung_Opacity/masks/Lung_Opacity-2652.png,True,False,299,299,L,106.64,367f85c45e149fd73f5d87875484f26c
Lung_Opacity-2653.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2653.png,Lung_Opacity/images/Lung_Opacity-2653.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2653.png,Lung_Opacity/masks/Lung_Opacity-2653.png,True,False,299,299,L,89.2,c225df8857b674f0c3c633abcad4d02b
Lung_Opacity-2654.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2654.png,Lung_Opacity/images/Lung_Opacity-2654.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2654.png,Lung_Opacity/masks/Lung_Opacity-2654.png,True,False,299,299,L,123.52,f398841648f83588d220f34e8b09da23
Lung_Opacity-2655.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2655.png,Lung_Opacity/images/Lung_Opacity-2655.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2655.png,Lung_Opacity/masks/Lung_Opacity-2655.png,True,False,299,299,L,112.86,caf0c10573644f6be3453e849bf2e652
Lung_Opacity-2656.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2656.png,Lung_Opacity/images/Lung_Opacity-2656.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2656.png,Lung_Opacity/masks/Lung_Opacity-2656.png,True,False,299,299,L,122.0,821d75e4a7b4d496193c25b365503c64
Lung_Opacity-2657.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2657.png,Lung_Opacity/images/Lung_Opacity-2657.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2657.png,Lung_Opacity/masks/Lung_Opacity-2657.png,True,False,299,299,L,136.02,7d8c517ba8d47b964b4d7c1e67bf9be1
Lung_Opacity-2658.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2658.png,Lung_Opacity/images/Lung_Opacity-2658.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2658.png,Lung_Opacity/masks/Lung_Opacity-2658.png,True,False,299,299,L,124.36,8dd1bd68e2d99e4f18ec44b14e8f9c11
Lung_Opacity-2659.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2659.png,Lung_Opacity/images/Lung_Opacity-2659.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2659.png,Lung_Opacity/masks/Lung_Opacity-2659.png,True,False,299,299,L,120.61,12e9bd856c528cfd35aefdab62c377f2
Lung_Opacity-266.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-266.png,Lung_Opacity/images/Lung_Opacity-266.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-266.png,Lung_Opacity/masks/Lung_Opacity-266.png,True,False,299,299,L,115.98,a8478817f6237584446578a78b4ed492
Lung_Opacity-2660.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2660.png,Lung_Opacity/images/Lung_Opacity-2660.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2660.png,Lung_Opacity/masks/Lung_Opacity-2660.png,True,False,299,299,L,112.86,8a69330f991cde7237b1c9e55de74117
Lung_Opacity-2661.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2661.png,Lung_Opacity/images/Lung_Opacity-2661.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2661.png,Lung_Opacity/masks/Lung_Opacity-2661.png,True,False,299,299,L,119.96,4adf28c45e25cfc0b1f8af722d2e84b0
Lung_Opacity-2662.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2662.png,Lung_Opacity/images/Lung_Opacity-2662.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2662.png,Lung_Opacity/masks/Lung_Opacity-2662.png,True,False,299,299,L,126.01,31b786628d7b7f8ea02718bdda46e200
Lung_Opacity-2663.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2663.png,Lung_Opacity/images/Lung_Opacity-2663.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2663.png,Lung_Opacity/masks/Lung_Opacity-2663.png,True,False,299,299,L,170.42,b379a8daeccefc831611e067bec2924a
Lung_Opacity-2664.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2664.png,Lung_Opacity/images/Lung_Opacity-2664.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2664.png,Lung_Opacity/masks/Lung_Opacity-2664.png,True,False,299,299,L,108.94,5596679858c836207d14c2bd22897806
Lung_Opacity-2665.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2665.png,Lung_Opacity/images/Lung_Opacity-2665.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2665.png,Lung_Opacity/masks/Lung_Opacity-2665.png,True,False,299,299,L,139.71,740cdead2a98056d3ea7aacb4eafdeab
Lung_Opacity-2666.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2666.png,Lung_Opacity/images/Lung_Opacity-2666.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2666.png,Lung_Opacity/masks/Lung_Opacity-2666.png,True,False,299,299,L,159.08,484559ba2711de33d59e4bb077c2fb6b
Lung_Opacity-2667.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2667.png,Lung_Opacity/images/Lung_Opacity-2667.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2667.png,Lung_Opacity/masks/Lung_Opacity-2667.png,True,False,299,299,L,100.12,52f1d69e285786b855c90541fe44b433
Lung_Opacity-2668.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2668.png,Lung_Opacity/images/Lung_Opacity-2668.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2668.png,Lung_Opacity/masks/Lung_Opacity-2668.png,True,False,299,299,L,154.53,10838b3c77c56ccccf7abd1f4bc407ea
Lung_Opacity-2669.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2669.png,Lung_Opacity/images/Lung_Opacity-2669.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2669.png,Lung_Opacity/masks/Lung_Opacity-2669.png,True,False,299,299,L,152.54,8ee5905651835adec7750bc7c7b6c43c
Lung_Opacity-267.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-267.png,Lung_Opacity/images/Lung_Opacity-267.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-267.png,Lung_Opacity/masks/Lung_Opacity-267.png,True,False,299,299,L,113.35,c6d43bb91d80eaf2f1213805ec91b223
Lung_Opacity-2670.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2670.png,Lung_Opacity/images/Lung_Opacity-2670.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2670.png,Lung_Opacity/masks/Lung_Opacity-2670.png,True,False,299,299,L,170.57,2b05a521c5e431fc46a980d6111e9423
Lung_Opacity-2671.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2671.png,Lung_Opacity/images/Lung_Opacity-2671.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2671.png,Lung_Opacity/masks/Lung_Opacity-2671.png,True,False,299,299,L,124.35,a19027a1bb370fc366d6a88dc1611697
Lung_Opacity-2672.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2672.png,Lung_Opacity/images/Lung_Opacity-2672.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2672.png,Lung_Opacity/masks/Lung_Opacity-2672.png,True,False,299,299,L,164.42,aac2dda758bc111355dc5adbf80fd847
Lung_Opacity-2673.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2673.png,Lung_Opacity/images/Lung_Opacity-2673.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2673.png,Lung_Opacity/masks/Lung_Opacity-2673.png,True,False,299,299,L,150.01,518214c46759116b83622e85534e4019
Lung_Opacity-2674.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2674.png,Lung_Opacity/images/Lung_Opacity-2674.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2674.png,Lung_Opacity/masks/Lung_Opacity-2674.png,True,False,299,299,L,117.75,7cc9bf679b272e2767cb0343a3a7b55d
Lung_Opacity-2675.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2675.png,Lung_Opacity/images/Lung_Opacity-2675.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2675.png,Lung_Opacity/masks/Lung_Opacity-2675.png,True,False,299,299,L,140.23,004e76905e14cb4f1234cd02eecf3f79
Lung_Opacity-2676.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2676.png,Lung_Opacity/images/Lung_Opacity-2676.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2676.png,Lung_Opacity/masks/Lung_Opacity-2676.png,True,False,299,299,L,146.08,30d2184646c7a41f0be937a93325fcfa
Lung_Opacity-2677.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2677.png,Lung_Opacity/images/Lung_Opacity-2677.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2677.png,Lung_Opacity/masks/Lung_Opacity-2677.png,True,False,299,299,L,156.87,9c315b409c688057be7b8e9bcefce70f
Lung_Opacity-2678.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2678.png,Lung_Opacity/images/Lung_Opacity-2678.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2678.png,Lung_Opacity/masks/Lung_Opacity-2678.png,True,False,299,299,L,111.95,c017d3b7533907d9e4ee655795e6b58f
Lung_Opacity-2679.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2679.png,Lung_Opacity/images/Lung_Opacity-2679.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2679.png,Lung_Opacity/masks/Lung_Opacity-2679.png,True,False,299,299,L,109.89,895353a305d7d783e5650e96f4d3339c
Lung_Opacity-268.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-268.png,Lung_Opacity/images/Lung_Opacity-268.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-268.png,Lung_Opacity/masks/Lung_Opacity-268.png,True,False,299,299,L,133.08,37f1d07201d302247f85741dd046a5cc
Lung_Opacity-2680.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2680.png,Lung_Opacity/images/Lung_Opacity-2680.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2680.png,Lung_Opacity/masks/Lung_Opacity-2680.png,True,False,299,299,L,126.4,9fe6dce89da59026f5ae64e76150a05e
Lung_Opacity-2681.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2681.png,Lung_Opacity/images/Lung_Opacity-2681.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2681.png,Lung_Opacity/masks/Lung_Opacity-2681.png,True,False,299,299,L,121.15,e40b073168fef11d6655d3ed67e53f4a
Lung_Opacity-2682.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2682.png,Lung_Opacity/images/Lung_Opacity-2682.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2682.png,Lung_Opacity/masks/Lung_Opacity-2682.png,True,False,299,299,L,98.68,be6efba575d4b80db13d2d67c80c6e19
Lung_Opacity-2683.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2683.png,Lung_Opacity/images/Lung_Opacity-2683.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2683.png,Lung_Opacity/masks/Lung_Opacity-2683.png,True,False,299,299,L,141.71,a52d0e1e16e5164a860529b42816e12e
Lung_Opacity-2684.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2684.png,Lung_Opacity/images/Lung_Opacity-2684.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2684.png,Lung_Opacity/masks/Lung_Opacity-2684.png,True,False,299,299,L,109.96,d7bc2f30232866af1792704e50ea8ef5
Lung_Opacity-2685.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2685.png,Lung_Opacity/images/Lung_Opacity-2685.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2685.png,Lung_Opacity/masks/Lung_Opacity-2685.png,True,False,299,299,L,124.24,8c0e08c96fcdce18bbc5c86ac9ce6018
Lung_Opacity-2686.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2686.png,Lung_Opacity/images/Lung_Opacity-2686.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2686.png,Lung_Opacity/masks/Lung_Opacity-2686.png,True,False,299,299,L,188.97,0848fff649bfe845e99709ca0820d289
Lung_Opacity-2687.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2687.png,Lung_Opacity/images/Lung_Opacity-2687.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2687.png,Lung_Opacity/masks/Lung_Opacity-2687.png,True,False,299,299,L,131.28,d40525d08cf2932e3f18098831d1b912
Lung_Opacity-2688.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2688.png,Lung_Opacity/images/Lung_Opacity-2688.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2688.png,Lung_Opacity/masks/Lung_Opacity-2688.png,True,False,299,299,L,147.3,2b031c0ecf425743aa45402ba9d79719
Lung_Opacity-2689.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2689.png,Lung_Opacity/images/Lung_Opacity-2689.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2689.png,Lung_Opacity/masks/Lung_Opacity-2689.png,True,False,299,299,L,128.64,7dbbe3f0233f041e21ae40a83ea4e01f
Lung_Opacity-269.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-269.png,Lung_Opacity/images/Lung_Opacity-269.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-269.png,Lung_Opacity/masks/Lung_Opacity-269.png,True,False,299,299,L,121.9,0d005236f031a98d69ac1f9085bb2b61
Lung_Opacity-2690.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2690.png,Lung_Opacity/images/Lung_Opacity-2690.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2690.png,Lung_Opacity/masks/Lung_Opacity-2690.png,True,False,299,299,L,113.46,6f276d525dc12e027ee7e22357bf5433
Lung_Opacity-2691.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2691.png,Lung_Opacity/images/Lung_Opacity-2691.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2691.png,Lung_Opacity/masks/Lung_Opacity-2691.png,True,False,299,299,L,85.28,8f97fe9996f6d6404e9611746bd0e3c2
Lung_Opacity-2692.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2692.png,Lung_Opacity/images/Lung_Opacity-2692.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2692.png,Lung_Opacity/masks/Lung_Opacity-2692.png,True,False,299,299,L,147.99,46506068a250363e54241385c871f2e2
Lung_Opacity-2693.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2693.png,Lung_Opacity/images/Lung_Opacity-2693.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2693.png,Lung_Opacity/masks/Lung_Opacity-2693.png,True,False,299,299,L,67.76,0fc393ed6c571a86da2b2e663207f655
Lung_Opacity-2694.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2694.png,Lung_Opacity/images/Lung_Opacity-2694.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2694.png,Lung_Opacity/masks/Lung_Opacity-2694.png,True,False,299,299,L,109.03,76656bf88d0fcb0562533a41db8d0852
Lung_Opacity-2695.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2695.png,Lung_Opacity/images/Lung_Opacity-2695.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2695.png,Lung_Opacity/masks/Lung_Opacity-2695.png,True,False,299,299,L,112.23,6ff798563351c54a71525cdb7590eff2
Lung_Opacity-2696.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2696.png,Lung_Opacity/images/Lung_Opacity-2696.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2696.png,Lung_Opacity/masks/Lung_Opacity-2696.png,True,False,299,299,L,143.91,3b8c6f1d3e8f95c626d24135616b8c06
Lung_Opacity-2697.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2697.png,Lung_Opacity/images/Lung_Opacity-2697.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2697.png,Lung_Opacity/masks/Lung_Opacity-2697.png,True,False,299,299,L,159.43,47cc8739191b68b40505879f20e33e87
Lung_Opacity-2698.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2698.png,Lung_Opacity/images/Lung_Opacity-2698.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2698.png,Lung_Opacity/masks/Lung_Opacity-2698.png,True,False,299,299,L,162.55,688574f5c0e9903485f9fb3469187643
Lung_Opacity-2699.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2699.png,Lung_Opacity/images/Lung_Opacity-2699.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2699.png,Lung_Opacity/masks/Lung_Opacity-2699.png,True,False,299,299,L,118.2,bcd8a90f1922a78816c674a1f53a9981
Lung_Opacity-27.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-27.png,Lung_Opacity/images/Lung_Opacity-27.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-27.png,Lung_Opacity/masks/Lung_Opacity-27.png,True,False,299,299,L,131.53,e2636963ebf7d7f553005e59c57c4396
Lung_Opacity-270.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-270.png,Lung_Opacity/images/Lung_Opacity-270.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-270.png,Lung_Opacity/masks/Lung_Opacity-270.png,True,False,299,299,L,113.27,fb5159a3a982c18269155c37fb304553
Lung_Opacity-2700.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2700.png,Lung_Opacity/images/Lung_Opacity-2700.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2700.png,Lung_Opacity/masks/Lung_Opacity-2700.png,True,False,299,299,L,121.73,6abe91c18818f7ee597146363f991f4a
Lung_Opacity-2701.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2701.png,Lung_Opacity/images/Lung_Opacity-2701.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2701.png,Lung_Opacity/masks/Lung_Opacity-2701.png,True,False,299,299,L,166.85,629206a4e822831f17a1957b4d8b9157
Lung_Opacity-2702.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2702.png,Lung_Opacity/images/Lung_Opacity-2702.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2702.png,Lung_Opacity/masks/Lung_Opacity-2702.png,True,False,299,299,L,105.69,23c25cf4b7011f239c4507b60e9575e6
Lung_Opacity-2703.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2703.png,Lung_Opacity/images/Lung_Opacity-2703.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2703.png,Lung_Opacity/masks/Lung_Opacity-2703.png,True,False,299,299,L,108.99,c7e802588f6b7831d7d7e1df40448ad6
Lung_Opacity-2704.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2704.png,Lung_Opacity/images/Lung_Opacity-2704.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2704.png,Lung_Opacity/masks/Lung_Opacity-2704.png,True,False,299,299,L,101.27,06b4447405f82d1533b281ae1c84b851
Lung_Opacity-2705.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2705.png,Lung_Opacity/images/Lung_Opacity-2705.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2705.png,Lung_Opacity/masks/Lung_Opacity-2705.png,True,False,299,299,L,160.46,2e59fe99a11a7e7e746c5bc77ee4b41f
Lung_Opacity-2706.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2706.png,Lung_Opacity/images/Lung_Opacity-2706.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2706.png,Lung_Opacity/masks/Lung_Opacity-2706.png,True,False,299,299,L,87.61,256c8bf4cbac5ffd9c8946b704159671
Lung_Opacity-2707.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2707.png,Lung_Opacity/images/Lung_Opacity-2707.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2707.png,Lung_Opacity/masks/Lung_Opacity-2707.png,True,False,299,299,L,73.26,49b3ba88018d71b8e5fe03794dece1d0
Lung_Opacity-2708.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2708.png,Lung_Opacity/images/Lung_Opacity-2708.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2708.png,Lung_Opacity/masks/Lung_Opacity-2708.png,True,False,299,299,L,157.57,24830f46758af060abaeb0655f8caf25
Lung_Opacity-2709.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2709.png,Lung_Opacity/images/Lung_Opacity-2709.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2709.png,Lung_Opacity/masks/Lung_Opacity-2709.png,True,False,299,299,L,112.82,8ba4560f520b51ab724f482d0ac1ecc8
Lung_Opacity-271.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-271.png,Lung_Opacity/images/Lung_Opacity-271.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-271.png,Lung_Opacity/masks/Lung_Opacity-271.png,True,False,299,299,L,176.93,af5a9fa05afd337a03ebc30359995999
Lung_Opacity-2710.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2710.png,Lung_Opacity/images/Lung_Opacity-2710.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2710.png,Lung_Opacity/masks/Lung_Opacity-2710.png,True,False,299,299,L,110.68,b3c39b8c16b3c2642e98894fe1376251
Lung_Opacity-2711.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2711.png,Lung_Opacity/images/Lung_Opacity-2711.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2711.png,Lung_Opacity/masks/Lung_Opacity-2711.png,True,False,299,299,L,123.0,01973af39f420445f7b1cccbf06a5d87
Lung_Opacity-2712.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2712.png,Lung_Opacity/images/Lung_Opacity-2712.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2712.png,Lung_Opacity/masks/Lung_Opacity-2712.png,True,False,299,299,L,172.38,a2dbe2f6fc02ba0d7bf2d6dddc07bc56
Lung_Opacity-2713.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2713.png,Lung_Opacity/images/Lung_Opacity-2713.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2713.png,Lung_Opacity/masks/Lung_Opacity-2713.png,True,False,299,299,L,158.43,a51b890a77273d0aa208c5db2582b339
Lung_Opacity-2714.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2714.png,Lung_Opacity/images/Lung_Opacity-2714.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2714.png,Lung_Opacity/masks/Lung_Opacity-2714.png,True,False,299,299,L,110.12,bf106b293e2d976f94c4a83abcd6983b
Lung_Opacity-2715.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2715.png,Lung_Opacity/images/Lung_Opacity-2715.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2715.png,Lung_Opacity/masks/Lung_Opacity-2715.png,True,False,299,299,L,111.19,beea29dad0a4a600146346a083a7cae4
Lung_Opacity-2716.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2716.png,Lung_Opacity/images/Lung_Opacity-2716.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2716.png,Lung_Opacity/masks/Lung_Opacity-2716.png,True,False,299,299,L,108.06,c06f4885db679ce653797ccb2cf0c4fb
Lung_Opacity-2717.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2717.png,Lung_Opacity/images/Lung_Opacity-2717.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2717.png,Lung_Opacity/masks/Lung_Opacity-2717.png,True,False,299,299,L,167.26,13e4373fe742b7edf2f27262dedaecc3
Lung_Opacity-2718.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2718.png,Lung_Opacity/images/Lung_Opacity-2718.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2718.png,Lung_Opacity/masks/Lung_Opacity-2718.png,True,False,299,299,L,108.86,1a7cabefdfee832d77fabfaae0110dee
Lung_Opacity-2719.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2719.png,Lung_Opacity/images/Lung_Opacity-2719.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2719.png,Lung_Opacity/masks/Lung_Opacity-2719.png,True,False,299,299,L,169.25,b7ec4c20c368b91406285b43796a3ba0
Lung_Opacity-272.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-272.png,Lung_Opacity/images/Lung_Opacity-272.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-272.png,Lung_Opacity/masks/Lung_Opacity-272.png,True,False,299,299,L,84.66,ee2179527654c7c3be1bdad743295113
Lung_Opacity-2720.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2720.png,Lung_Opacity/images/Lung_Opacity-2720.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2720.png,Lung_Opacity/masks/Lung_Opacity-2720.png,True,False,299,299,L,163.73,6184a166fe3d89afa87db5f6f322d04c
Lung_Opacity-2721.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2721.png,Lung_Opacity/images/Lung_Opacity-2721.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2721.png,Lung_Opacity/masks/Lung_Opacity-2721.png,True,False,299,299,L,117.89,c5c966b582b37ebbe9c547f5c2fb7f17
Lung_Opacity-2722.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2722.png,Lung_Opacity/images/Lung_Opacity-2722.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2722.png,Lung_Opacity/masks/Lung_Opacity-2722.png,True,False,299,299,L,153.14,bcba21a4d9d1219ba92f14401b78ecfb
Lung_Opacity-2723.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2723.png,Lung_Opacity/images/Lung_Opacity-2723.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2723.png,Lung_Opacity/masks/Lung_Opacity-2723.png,True,False,299,299,L,140.02,12438b40efbf1c0177a49f5c2c2802ba
Lung_Opacity-2724.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2724.png,Lung_Opacity/images/Lung_Opacity-2724.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2724.png,Lung_Opacity/masks/Lung_Opacity-2724.png,True,False,299,299,L,125.41,4fb63e3b89371e8a8cfb0897c5c3c9f0
Lung_Opacity-2725.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2725.png,Lung_Opacity/images/Lung_Opacity-2725.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2725.png,Lung_Opacity/masks/Lung_Opacity-2725.png,True,False,299,299,L,145.1,a9aa2e6be03acb8177275e14279c09ac
Lung_Opacity-2726.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2726.png,Lung_Opacity/images/Lung_Opacity-2726.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2726.png,Lung_Opacity/masks/Lung_Opacity-2726.png,True,False,299,299,L,112.05,6384bad496862a35aeeb2a7a7738e0cd
Lung_Opacity-2727.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2727.png,Lung_Opacity/images/Lung_Opacity-2727.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2727.png,Lung_Opacity/masks/Lung_Opacity-2727.png,True,False,299,299,L,113.74,a696501139e94c8fffbc0eccfbd5b4ae
Lung_Opacity-2728.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2728.png,Lung_Opacity/images/Lung_Opacity-2728.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2728.png,Lung_Opacity/masks/Lung_Opacity-2728.png,True,False,299,299,L,163.09,952fc470f4c92ff325e30578f303d4ec
Lung_Opacity-2729.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2729.png,Lung_Opacity/images/Lung_Opacity-2729.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2729.png,Lung_Opacity/masks/Lung_Opacity-2729.png,True,False,299,299,L,162.13,ab74ae0c85be47b7cf18d942bd8755e4
Lung_Opacity-273.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-273.png,Lung_Opacity/images/Lung_Opacity-273.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-273.png,Lung_Opacity/masks/Lung_Opacity-273.png,True,False,299,299,L,153.42,a84ad65cd624b2b7eff76fac8e8bb14e
Lung_Opacity-2730.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2730.png,Lung_Opacity/images/Lung_Opacity-2730.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2730.png,Lung_Opacity/masks/Lung_Opacity-2730.png,True,False,299,299,L,117.76,a4c70686cf0c3e0d481681b55cadf08c
Lung_Opacity-2731.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2731.png,Lung_Opacity/images/Lung_Opacity-2731.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2731.png,Lung_Opacity/masks/Lung_Opacity-2731.png,True,False,299,299,L,120.0,d0e5adae9da91b6ecd0f6f9bd8f12d41
Lung_Opacity-2732.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2732.png,Lung_Opacity/images/Lung_Opacity-2732.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2732.png,Lung_Opacity/masks/Lung_Opacity-2732.png,True,False,299,299,L,124.09,7a62bfcf2558eac4e4f938db97360b6a
Lung_Opacity-2733.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2733.png,Lung_Opacity/images/Lung_Opacity-2733.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2733.png,Lung_Opacity/masks/Lung_Opacity-2733.png,True,False,299,299,L,130.53,ef6a41187a0fed54cc574bba5dccb817
Lung_Opacity-2734.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2734.png,Lung_Opacity/images/Lung_Opacity-2734.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2734.png,Lung_Opacity/masks/Lung_Opacity-2734.png,True,False,299,299,L,113.17,9842f07e4f916c6fd4b3da373b4c4d12
Lung_Opacity-2735.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2735.png,Lung_Opacity/images/Lung_Opacity-2735.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2735.png,Lung_Opacity/masks/Lung_Opacity-2735.png,True,False,299,299,L,108.34,22a75e6f7a91b6491c921bdae1b7d38a
Lung_Opacity-2736.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2736.png,Lung_Opacity/images/Lung_Opacity-2736.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2736.png,Lung_Opacity/masks/Lung_Opacity-2736.png,True,False,299,299,L,124.98,836d9cdfc79585c7d233532e668ee450
Lung_Opacity-2737.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2737.png,Lung_Opacity/images/Lung_Opacity-2737.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2737.png,Lung_Opacity/masks/Lung_Opacity-2737.png,True,False,299,299,L,146.55,f5ab7bb1f81f1a917e4d6c98b29a51e8
Lung_Opacity-2738.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2738.png,Lung_Opacity/images/Lung_Opacity-2738.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2738.png,Lung_Opacity/masks/Lung_Opacity-2738.png,True,False,299,299,L,147.89,0b969392e38cc10ef92af38e3508c00c
Lung_Opacity-2739.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2739.png,Lung_Opacity/images/Lung_Opacity-2739.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2739.png,Lung_Opacity/masks/Lung_Opacity-2739.png,True,False,299,299,L,134.13,27f7de1ed6c4e0229a850ba146e46db0
Lung_Opacity-274.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-274.png,Lung_Opacity/images/Lung_Opacity-274.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-274.png,Lung_Opacity/masks/Lung_Opacity-274.png,True,False,299,299,L,151.12,04c7743c486090969bf91edb51451188
Lung_Opacity-2740.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2740.png,Lung_Opacity/images/Lung_Opacity-2740.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2740.png,Lung_Opacity/masks/Lung_Opacity-2740.png,True,False,299,299,L,155.07,36bd32c5f51689a676ed1d263f3089ea
Lung_Opacity-2741.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2741.png,Lung_Opacity/images/Lung_Opacity-2741.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2741.png,Lung_Opacity/masks/Lung_Opacity-2741.png,True,False,299,299,L,115.26,bd0c9aadfdbc67c0ec944037163b34c7
Lung_Opacity-2742.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2742.png,Lung_Opacity/images/Lung_Opacity-2742.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2742.png,Lung_Opacity/masks/Lung_Opacity-2742.png,True,False,299,299,L,127.22,97633d1db4a08d369f4a2b24fae7beba
Lung_Opacity-2743.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2743.png,Lung_Opacity/images/Lung_Opacity-2743.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2743.png,Lung_Opacity/masks/Lung_Opacity-2743.png,True,False,299,299,L,107.52,efb0e3704f82a1955deebf7b00032032
Lung_Opacity-2744.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2744.png,Lung_Opacity/images/Lung_Opacity-2744.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2744.png,Lung_Opacity/masks/Lung_Opacity-2744.png,True,False,299,299,L,148.63,5a2a05e9b4fbf9801defba8d2e8cb966
Lung_Opacity-2745.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2745.png,Lung_Opacity/images/Lung_Opacity-2745.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2745.png,Lung_Opacity/masks/Lung_Opacity-2745.png,True,False,299,299,L,126.36,3035f5f37f5e5d0fd1e2ef677d68ab1b
Lung_Opacity-2746.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2746.png,Lung_Opacity/images/Lung_Opacity-2746.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2746.png,Lung_Opacity/masks/Lung_Opacity-2746.png,True,False,299,299,L,167.06,6938679ef90679429ceffbbd6d903a48
Lung_Opacity-2747.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2747.png,Lung_Opacity/images/Lung_Opacity-2747.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2747.png,Lung_Opacity/masks/Lung_Opacity-2747.png,True,False,299,299,L,133.25,e29d318ed7d62f8b450b0fade6bbb861
Lung_Opacity-2748.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2748.png,Lung_Opacity/images/Lung_Opacity-2748.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2748.png,Lung_Opacity/masks/Lung_Opacity-2748.png,True,False,299,299,L,118.93,e88b6d2f6e7fe2a444ef21a187af155c
Lung_Opacity-2749.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2749.png,Lung_Opacity/images/Lung_Opacity-2749.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2749.png,Lung_Opacity/masks/Lung_Opacity-2749.png,True,False,299,299,L,113.79,ae24e8588866d08360df90882ac25728
Lung_Opacity-275.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-275.png,Lung_Opacity/images/Lung_Opacity-275.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-275.png,Lung_Opacity/masks/Lung_Opacity-275.png,True,False,299,299,L,157.02,2c205c9e13949706d9391652913d2115
Lung_Opacity-2750.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2750.png,Lung_Opacity/images/Lung_Opacity-2750.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2750.png,Lung_Opacity/masks/Lung_Opacity-2750.png,True,False,299,299,L,119.28,858be298ca8cdc2d6a1ad7cb9630efc3
Lung_Opacity-2751.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2751.png,Lung_Opacity/images/Lung_Opacity-2751.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2751.png,Lung_Opacity/masks/Lung_Opacity-2751.png,True,False,299,299,L,91.37,a7c2539446c18d5a7a2f9e7f01a42432
Lung_Opacity-2752.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2752.png,Lung_Opacity/images/Lung_Opacity-2752.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2752.png,Lung_Opacity/masks/Lung_Opacity-2752.png,True,False,299,299,L,95.32,d19d2ca87e7dee22fa4e20ba4f8b0284
Lung_Opacity-2753.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2753.png,Lung_Opacity/images/Lung_Opacity-2753.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2753.png,Lung_Opacity/masks/Lung_Opacity-2753.png,True,False,299,299,L,145.68,846eb47a853d1428761bad3a3dc8becb
Lung_Opacity-2754.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2754.png,Lung_Opacity/images/Lung_Opacity-2754.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2754.png,Lung_Opacity/masks/Lung_Opacity-2754.png,True,False,299,299,L,167.92,6b5ac272138cf4e93ad7cd5cfed82b53
Lung_Opacity-2755.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2755.png,Lung_Opacity/images/Lung_Opacity-2755.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2755.png,Lung_Opacity/masks/Lung_Opacity-2755.png,True,False,299,299,L,116.14,1120843dafe5f0a8333c8a348160583f
Lung_Opacity-2756.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2756.png,Lung_Opacity/images/Lung_Opacity-2756.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2756.png,Lung_Opacity/masks/Lung_Opacity-2756.png,True,False,299,299,L,141.85,c78946eeee7f048dc70945adfda6052b
Lung_Opacity-2757.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2757.png,Lung_Opacity/images/Lung_Opacity-2757.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2757.png,Lung_Opacity/masks/Lung_Opacity-2757.png,True,False,299,299,L,167.97,a9679a0dab784fa916da3173a611c4d8
Lung_Opacity-2758.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2758.png,Lung_Opacity/images/Lung_Opacity-2758.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2758.png,Lung_Opacity/masks/Lung_Opacity-2758.png,True,False,299,299,L,117.81,fdbe17f6312014cd226f07f802ca2672
Lung_Opacity-2759.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2759.png,Lung_Opacity/images/Lung_Opacity-2759.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2759.png,Lung_Opacity/masks/Lung_Opacity-2759.png,True,False,299,299,L,166.66,e90daf4bb4d1300d10586392ecc73d00
Lung_Opacity-276.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-276.png,Lung_Opacity/images/Lung_Opacity-276.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-276.png,Lung_Opacity/masks/Lung_Opacity-276.png,True,False,299,299,L,122.07,baa68482b749ae6cfa313b3ab057dd59
Lung_Opacity-2760.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2760.png,Lung_Opacity/images/Lung_Opacity-2760.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2760.png,Lung_Opacity/masks/Lung_Opacity-2760.png,True,False,299,299,L,95.35,577f64a5183744597fa00c760136fc6d
Lung_Opacity-2761.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2761.png,Lung_Opacity/images/Lung_Opacity-2761.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2761.png,Lung_Opacity/masks/Lung_Opacity-2761.png,True,False,299,299,L,116.15,b19553c6816cf6a332f32c6eac4c7907
Lung_Opacity-2762.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2762.png,Lung_Opacity/images/Lung_Opacity-2762.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2762.png,Lung_Opacity/masks/Lung_Opacity-2762.png,True,False,299,299,L,118.85,d4718cc727946956c54284ba7ab3f18c
Lung_Opacity-2763.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2763.png,Lung_Opacity/images/Lung_Opacity-2763.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2763.png,Lung_Opacity/masks/Lung_Opacity-2763.png,True,False,299,299,L,151.4,e1fb322ed964763a7b88ff63ee70824a
Lung_Opacity-2764.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2764.png,Lung_Opacity/images/Lung_Opacity-2764.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2764.png,Lung_Opacity/masks/Lung_Opacity-2764.png,True,False,299,299,L,130.91,4e44420f9f227eef4a62ff0c4018e292
Lung_Opacity-2765.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2765.png,Lung_Opacity/images/Lung_Opacity-2765.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2765.png,Lung_Opacity/masks/Lung_Opacity-2765.png,True,False,299,299,L,122.11,2320a513d1c7199d953e8627ddfce682
Lung_Opacity-2766.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2766.png,Lung_Opacity/images/Lung_Opacity-2766.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2766.png,Lung_Opacity/masks/Lung_Opacity-2766.png,True,False,299,299,L,114.85,bd81fff419f68ff46808cde690f78fb6
Lung_Opacity-2767.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2767.png,Lung_Opacity/images/Lung_Opacity-2767.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2767.png,Lung_Opacity/masks/Lung_Opacity-2767.png,True,False,299,299,L,109.9,d5b3726dc4a71b3c8130912fc73463b9
Lung_Opacity-2768.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2768.png,Lung_Opacity/images/Lung_Opacity-2768.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2768.png,Lung_Opacity/masks/Lung_Opacity-2768.png,True,False,299,299,L,102.3,91774382ee51819d4bf434e5d40b43d2
Lung_Opacity-2769.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2769.png,Lung_Opacity/images/Lung_Opacity-2769.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2769.png,Lung_Opacity/masks/Lung_Opacity-2769.png,True,False,299,299,L,136.05,e3caff87540444c69190bd08940b2ae0
Lung_Opacity-277.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-277.png,Lung_Opacity/images/Lung_Opacity-277.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-277.png,Lung_Opacity/masks/Lung_Opacity-277.png,True,False,299,299,L,136.42,ef7434291a538a022ed8350de0c88661
Lung_Opacity-2770.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2770.png,Lung_Opacity/images/Lung_Opacity-2770.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2770.png,Lung_Opacity/masks/Lung_Opacity-2770.png,True,False,299,299,L,158.24,7d653f27d2c5501d4b6856309c4f1606
Lung_Opacity-2771.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2771.png,Lung_Opacity/images/Lung_Opacity-2771.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2771.png,Lung_Opacity/masks/Lung_Opacity-2771.png,True,False,299,299,L,93.22,929c9249051302c81487ce273a68316a
Lung_Opacity-2772.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2772.png,Lung_Opacity/images/Lung_Opacity-2772.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2772.png,Lung_Opacity/masks/Lung_Opacity-2772.png,True,False,299,299,L,163.21,bbbab138f1b60a149e91ce3e5f97d591
Lung_Opacity-2773.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2773.png,Lung_Opacity/images/Lung_Opacity-2773.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2773.png,Lung_Opacity/masks/Lung_Opacity-2773.png,True,False,299,299,L,166.3,e60b21b25f25e911895a713f84746f1b
Lung_Opacity-2774.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2774.png,Lung_Opacity/images/Lung_Opacity-2774.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2774.png,Lung_Opacity/masks/Lung_Opacity-2774.png,True,False,299,299,L,124.38,cc09bdf7728a8d4975341580fa62de14
Lung_Opacity-2775.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2775.png,Lung_Opacity/images/Lung_Opacity-2775.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2775.png,Lung_Opacity/masks/Lung_Opacity-2775.png,True,False,299,299,L,100.05,68e32c70dd8350a3d06ac6256e12b0fe
Lung_Opacity-2776.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2776.png,Lung_Opacity/images/Lung_Opacity-2776.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2776.png,Lung_Opacity/masks/Lung_Opacity-2776.png,True,False,299,299,L,140.07,7a5801d8fa7c1df7b5b6f01a60348dcb
Lung_Opacity-2777.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2777.png,Lung_Opacity/images/Lung_Opacity-2777.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2777.png,Lung_Opacity/masks/Lung_Opacity-2777.png,True,False,299,299,L,116.98,edfd3c62d85134f78460f38a04a5472b
Lung_Opacity-2778.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2778.png,Lung_Opacity/images/Lung_Opacity-2778.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2778.png,Lung_Opacity/masks/Lung_Opacity-2778.png,True,False,299,299,L,111.04,cd80a66c87a72aec9ef2ce8294f9c006
Lung_Opacity-2779.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2779.png,Lung_Opacity/images/Lung_Opacity-2779.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2779.png,Lung_Opacity/masks/Lung_Opacity-2779.png,True,False,299,299,L,112.78,774241efe038b36036173690aa1b9095
Lung_Opacity-278.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-278.png,Lung_Opacity/images/Lung_Opacity-278.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-278.png,Lung_Opacity/masks/Lung_Opacity-278.png,True,False,299,299,L,110.21,ab7db8970c6697f37c49bbe251b9baba
Lung_Opacity-2780.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2780.png,Lung_Opacity/images/Lung_Opacity-2780.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2780.png,Lung_Opacity/masks/Lung_Opacity-2780.png,True,False,299,299,L,152.62,d04b028a7818cbc77806911f0e76e3d5
Lung_Opacity-2781.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2781.png,Lung_Opacity/images/Lung_Opacity-2781.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2781.png,Lung_Opacity/masks/Lung_Opacity-2781.png,True,False,299,299,L,157.64,f2fd66e16e0125aadec7d748fa137632
Lung_Opacity-2782.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2782.png,Lung_Opacity/images/Lung_Opacity-2782.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2782.png,Lung_Opacity/masks/Lung_Opacity-2782.png,True,False,299,299,L,172.09,e84b65cf756412cf36b3cd9ae0b45694
Lung_Opacity-2783.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2783.png,Lung_Opacity/images/Lung_Opacity-2783.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2783.png,Lung_Opacity/masks/Lung_Opacity-2783.png,True,False,299,299,L,109.02,f25731313103ab544dccda37a89ac5d7
Lung_Opacity-2784.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2784.png,Lung_Opacity/images/Lung_Opacity-2784.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2784.png,Lung_Opacity/masks/Lung_Opacity-2784.png,True,False,299,299,L,155.42,fea4818d82238c4b2304d839e9b6bbe3
Lung_Opacity-2785.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2785.png,Lung_Opacity/images/Lung_Opacity-2785.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2785.png,Lung_Opacity/masks/Lung_Opacity-2785.png,True,False,299,299,L,116.63,b70b5be0b5f59217a2f46ae4548d634f
Lung_Opacity-2786.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2786.png,Lung_Opacity/images/Lung_Opacity-2786.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2786.png,Lung_Opacity/masks/Lung_Opacity-2786.png,True,False,299,299,L,144.99,3549541b6187c4388f69a470fd156dd9
Lung_Opacity-2787.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2787.png,Lung_Opacity/images/Lung_Opacity-2787.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2787.png,Lung_Opacity/masks/Lung_Opacity-2787.png,True,False,299,299,L,129.84,7c31072aebc2f9a043edcc750cd95524
Lung_Opacity-2788.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2788.png,Lung_Opacity/images/Lung_Opacity-2788.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2788.png,Lung_Opacity/masks/Lung_Opacity-2788.png,True,False,299,299,L,158.58,1b7de23766ac886eddcdae9bccb1e70e
Lung_Opacity-2789.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2789.png,Lung_Opacity/images/Lung_Opacity-2789.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2789.png,Lung_Opacity/masks/Lung_Opacity-2789.png,True,False,299,299,L,147.91,552c9af575c5b0c81e7a55596a6f90a6
Lung_Opacity-279.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-279.png,Lung_Opacity/images/Lung_Opacity-279.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-279.png,Lung_Opacity/masks/Lung_Opacity-279.png,True,False,299,299,L,105.07,0883d2b42f15a2efe8cd856f5c00b2c1
Lung_Opacity-2790.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2790.png,Lung_Opacity/images/Lung_Opacity-2790.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2790.png,Lung_Opacity/masks/Lung_Opacity-2790.png,True,False,299,299,L,147.84,415417073660ba9aef96a10599b36c9d
Lung_Opacity-2791.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2791.png,Lung_Opacity/images/Lung_Opacity-2791.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2791.png,Lung_Opacity/masks/Lung_Opacity-2791.png,True,False,299,299,L,110.59,df79db3354a2ac114eb2cd5bed0b5412
Lung_Opacity-2792.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2792.png,Lung_Opacity/images/Lung_Opacity-2792.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2792.png,Lung_Opacity/masks/Lung_Opacity-2792.png,True,False,299,299,L,137.86,cf3548de02c176f32d22f8a527518e07
Lung_Opacity-2793.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2793.png,Lung_Opacity/images/Lung_Opacity-2793.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2793.png,Lung_Opacity/masks/Lung_Opacity-2793.png,True,False,299,299,L,127.06,d64f147f7cccdacbe2bf3075e23ace84
Lung_Opacity-2794.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2794.png,Lung_Opacity/images/Lung_Opacity-2794.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2794.png,Lung_Opacity/masks/Lung_Opacity-2794.png,True,False,299,299,L,110.39,9120cc97aa548aa08e2940c75571de07
Lung_Opacity-2795.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2795.png,Lung_Opacity/images/Lung_Opacity-2795.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2795.png,Lung_Opacity/masks/Lung_Opacity-2795.png,True,False,299,299,L,105.3,4043255f31101087502640adeedf9009
Lung_Opacity-2796.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2796.png,Lung_Opacity/images/Lung_Opacity-2796.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2796.png,Lung_Opacity/masks/Lung_Opacity-2796.png,True,False,299,299,L,140.83,bf5b3eff2202231bdc091f18c53dbec7
Lung_Opacity-2797.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2797.png,Lung_Opacity/images/Lung_Opacity-2797.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2797.png,Lung_Opacity/masks/Lung_Opacity-2797.png,True,False,299,299,L,124.34,db21949942c3ca39ea60f132e2b01d05
Lung_Opacity-2798.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2798.png,Lung_Opacity/images/Lung_Opacity-2798.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2798.png,Lung_Opacity/masks/Lung_Opacity-2798.png,True,False,299,299,L,100.13,87b7efb8a8a30797d7cbaf96513406e3
Lung_Opacity-2799.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2799.png,Lung_Opacity/images/Lung_Opacity-2799.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2799.png,Lung_Opacity/masks/Lung_Opacity-2799.png,True,False,299,299,L,87.96,b7c8b2df843667e0108dc8b03cc7dbce
Lung_Opacity-28.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-28.png,Lung_Opacity/images/Lung_Opacity-28.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-28.png,Lung_Opacity/masks/Lung_Opacity-28.png,True,False,299,299,L,66.2,eba65afb8f040d32afce5cbdf1d8840c
Lung_Opacity-280.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-280.png,Lung_Opacity/images/Lung_Opacity-280.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-280.png,Lung_Opacity/masks/Lung_Opacity-280.png,True,False,299,299,L,157.24,343c6591df58c3dec0c9433e90854ece
Lung_Opacity-2800.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2800.png,Lung_Opacity/images/Lung_Opacity-2800.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2800.png,Lung_Opacity/masks/Lung_Opacity-2800.png,True,False,299,299,L,172.4,c5f63993cd53d7839028ecde92ef78ef
Lung_Opacity-2801.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2801.png,Lung_Opacity/images/Lung_Opacity-2801.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2801.png,Lung_Opacity/masks/Lung_Opacity-2801.png,True,False,299,299,L,115.84,f8c35e2556403c117e8738550e18ef63
Lung_Opacity-2802.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2802.png,Lung_Opacity/images/Lung_Opacity-2802.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2802.png,Lung_Opacity/masks/Lung_Opacity-2802.png,True,False,299,299,L,163.34,4fb8e7334b8056460c723a4720c92d8a
Lung_Opacity-2803.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2803.png,Lung_Opacity/images/Lung_Opacity-2803.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2803.png,Lung_Opacity/masks/Lung_Opacity-2803.png,True,False,299,299,L,116.28,4b840704ec2c6daa11989a095e2b70fc
Lung_Opacity-2804.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2804.png,Lung_Opacity/images/Lung_Opacity-2804.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2804.png,Lung_Opacity/masks/Lung_Opacity-2804.png,True,False,299,299,L,168.96,3d5a02723fcb96b2ee5197abe9558948
Lung_Opacity-2805.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2805.png,Lung_Opacity/images/Lung_Opacity-2805.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2805.png,Lung_Opacity/masks/Lung_Opacity-2805.png,True,False,299,299,L,133.75,b21afd3fd1b5f3dd7ca9e43dfdc96b3d
Lung_Opacity-2806.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2806.png,Lung_Opacity/images/Lung_Opacity-2806.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2806.png,Lung_Opacity/masks/Lung_Opacity-2806.png,True,False,299,299,L,134.8,380fd9fa4bc86993667f9a7f8c0cd35e
Lung_Opacity-2807.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2807.png,Lung_Opacity/images/Lung_Opacity-2807.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2807.png,Lung_Opacity/masks/Lung_Opacity-2807.png,True,False,299,299,L,114.71,492ddb87f8d4c5ff1be4dd861e7867f4
Lung_Opacity-2808.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2808.png,Lung_Opacity/images/Lung_Opacity-2808.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2808.png,Lung_Opacity/masks/Lung_Opacity-2808.png,True,False,299,299,L,115.78,28941ae54e33be5cc89f4670417f7770
Lung_Opacity-2809.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2809.png,Lung_Opacity/images/Lung_Opacity-2809.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2809.png,Lung_Opacity/masks/Lung_Opacity-2809.png,True,False,299,299,L,150.54,2ce640303966e1e5b2e3f064e75ba683
Lung_Opacity-281.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-281.png,Lung_Opacity/images/Lung_Opacity-281.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-281.png,Lung_Opacity/masks/Lung_Opacity-281.png,True,False,299,299,L,135.29,5c9262c1f458369b6f74678c2577c92d
Lung_Opacity-2810.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2810.png,Lung_Opacity/images/Lung_Opacity-2810.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2810.png,Lung_Opacity/masks/Lung_Opacity-2810.png,True,False,299,299,L,107.98,5d53810b0097a38bcdf20b28c2e2edc2
Lung_Opacity-2811.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2811.png,Lung_Opacity/images/Lung_Opacity-2811.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2811.png,Lung_Opacity/masks/Lung_Opacity-2811.png,True,False,299,299,L,96.1,1e2e3a879a746abd92e358916c6eae90
Lung_Opacity-2812.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2812.png,Lung_Opacity/images/Lung_Opacity-2812.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2812.png,Lung_Opacity/masks/Lung_Opacity-2812.png,True,False,299,299,L,62.51,9b06767f598be4ea32ede13114520e3d
Lung_Opacity-2813.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2813.png,Lung_Opacity/images/Lung_Opacity-2813.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2813.png,Lung_Opacity/masks/Lung_Opacity-2813.png,True,False,299,299,L,134.27,5f0ee8251190937fdae9ea7af6e9a97c
Lung_Opacity-2814.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2814.png,Lung_Opacity/images/Lung_Opacity-2814.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2814.png,Lung_Opacity/masks/Lung_Opacity-2814.png,True,False,299,299,L,130.48,a3a2a01ce3efff7a598c0ee4b2108c6d
Lung_Opacity-2815.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2815.png,Lung_Opacity/images/Lung_Opacity-2815.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2815.png,Lung_Opacity/masks/Lung_Opacity-2815.png,True,False,299,299,L,158.55,8ede44979c111baa1626d7bc8ca39f7c
Lung_Opacity-2816.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2816.png,Lung_Opacity/images/Lung_Opacity-2816.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2816.png,Lung_Opacity/masks/Lung_Opacity-2816.png,True,False,299,299,L,112.98,212224ac171adf92076d75203fb29e97
Lung_Opacity-2817.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2817.png,Lung_Opacity/images/Lung_Opacity-2817.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2817.png,Lung_Opacity/masks/Lung_Opacity-2817.png,True,False,299,299,L,123.81,6ce58b8c5d16c06e8b9fcd0f67929a70
Lung_Opacity-2818.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2818.png,Lung_Opacity/images/Lung_Opacity-2818.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2818.png,Lung_Opacity/masks/Lung_Opacity-2818.png,True,False,299,299,L,118.89,604854726fbe4c2bccac8781f1e3c083
Lung_Opacity-2819.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2819.png,Lung_Opacity/images/Lung_Opacity-2819.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2819.png,Lung_Opacity/masks/Lung_Opacity-2819.png,True,False,299,299,L,141.06,645a7d1e1784e72a0ede957543c14c1a
Lung_Opacity-282.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-282.png,Lung_Opacity/images/Lung_Opacity-282.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-282.png,Lung_Opacity/masks/Lung_Opacity-282.png,True,False,299,299,L,115.67,45b4f4e8bde33c94659eac513ba9c7ee
Lung_Opacity-2820.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2820.png,Lung_Opacity/images/Lung_Opacity-2820.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2820.png,Lung_Opacity/masks/Lung_Opacity-2820.png,True,False,299,299,L,111.22,e754177cc19211816aaa33e8d29c06a1
Lung_Opacity-2821.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2821.png,Lung_Opacity/images/Lung_Opacity-2821.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2821.png,Lung_Opacity/masks/Lung_Opacity-2821.png,True,False,299,299,L,130.83,03528fb2b73ee38f43a1d2f2d6e2d3e0
Lung_Opacity-2822.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2822.png,Lung_Opacity/images/Lung_Opacity-2822.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2822.png,Lung_Opacity/masks/Lung_Opacity-2822.png,True,False,299,299,L,151.48,b58647f74ab43d214fe6942178150c55
Lung_Opacity-2823.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2823.png,Lung_Opacity/images/Lung_Opacity-2823.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2823.png,Lung_Opacity/masks/Lung_Opacity-2823.png,True,False,299,299,L,109.77,886ba5115fc8a0efac65ac29c4bd6d85
Lung_Opacity-2824.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2824.png,Lung_Opacity/images/Lung_Opacity-2824.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2824.png,Lung_Opacity/masks/Lung_Opacity-2824.png,True,False,299,299,L,111.98,ceca8a643b008e5544f687ddb26f2521
Lung_Opacity-2825.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2825.png,Lung_Opacity/images/Lung_Opacity-2825.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2825.png,Lung_Opacity/masks/Lung_Opacity-2825.png,True,False,299,299,L,157.38,7de8c734f382b7cd150a5ed3c5280e8b
Lung_Opacity-2826.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2826.png,Lung_Opacity/images/Lung_Opacity-2826.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2826.png,Lung_Opacity/masks/Lung_Opacity-2826.png,True,False,299,299,L,120.14,250be0fcfe956662b269eff50de6ee7b
Lung_Opacity-2827.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2827.png,Lung_Opacity/images/Lung_Opacity-2827.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2827.png,Lung_Opacity/masks/Lung_Opacity-2827.png,True,False,299,299,L,85.73,062e54dd231be618da3e2be09cc830b6
Lung_Opacity-2828.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2828.png,Lung_Opacity/images/Lung_Opacity-2828.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2828.png,Lung_Opacity/masks/Lung_Opacity-2828.png,True,False,299,299,L,174.49,54f58be3cc58889fbf9ed4f9ce901b4a
Lung_Opacity-2829.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2829.png,Lung_Opacity/images/Lung_Opacity-2829.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2829.png,Lung_Opacity/masks/Lung_Opacity-2829.png,True,False,299,299,L,114.65,66a317ffca79e27f180dfc5024f6b2fc
Lung_Opacity-283.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-283.png,Lung_Opacity/images/Lung_Opacity-283.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-283.png,Lung_Opacity/masks/Lung_Opacity-283.png,True,False,299,299,L,103.54,02384553019ade1fe3437ce633d99923
Lung_Opacity-2830.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2830.png,Lung_Opacity/images/Lung_Opacity-2830.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2830.png,Lung_Opacity/masks/Lung_Opacity-2830.png,True,False,299,299,L,128.62,9073d3c7bfd9df23f1c5183271dbc023
Lung_Opacity-2831.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2831.png,Lung_Opacity/images/Lung_Opacity-2831.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2831.png,Lung_Opacity/masks/Lung_Opacity-2831.png,True,False,299,299,L,138.97,00f09f38903ecb09fe2ba12b5411dc9e
Lung_Opacity-2832.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2832.png,Lung_Opacity/images/Lung_Opacity-2832.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2832.png,Lung_Opacity/masks/Lung_Opacity-2832.png,True,False,299,299,L,117.92,2d38c142b68a6120dadf23e9c9b872de
Lung_Opacity-2833.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2833.png,Lung_Opacity/images/Lung_Opacity-2833.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2833.png,Lung_Opacity/masks/Lung_Opacity-2833.png,True,False,299,299,L,108.05,70e44dc9e23b3c3c09505a07ef96e513
Lung_Opacity-2834.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2834.png,Lung_Opacity/images/Lung_Opacity-2834.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2834.png,Lung_Opacity/masks/Lung_Opacity-2834.png,True,False,299,299,L,108.72,62f233ae1ce4b91285a18df5aa8a67dd
Lung_Opacity-2835.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2835.png,Lung_Opacity/images/Lung_Opacity-2835.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2835.png,Lung_Opacity/masks/Lung_Opacity-2835.png,True,False,299,299,L,83.86,cf15565b70a4059a14283dcd28a873f8
Lung_Opacity-2836.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2836.png,Lung_Opacity/images/Lung_Opacity-2836.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2836.png,Lung_Opacity/masks/Lung_Opacity-2836.png,True,False,299,299,L,168.98,88894b9c835860052fd05c9e1d58554f
Lung_Opacity-2837.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2837.png,Lung_Opacity/images/Lung_Opacity-2837.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2837.png,Lung_Opacity/masks/Lung_Opacity-2837.png,True,False,299,299,L,123.05,6f1afd4904d4ba090bf7e0882c68573e
Lung_Opacity-2838.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2838.png,Lung_Opacity/images/Lung_Opacity-2838.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2838.png,Lung_Opacity/masks/Lung_Opacity-2838.png,True,False,299,299,L,129.82,4fc7d687b3fc2eed59fb05965ac64ed9
Lung_Opacity-2839.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2839.png,Lung_Opacity/images/Lung_Opacity-2839.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2839.png,Lung_Opacity/masks/Lung_Opacity-2839.png,True,False,299,299,L,153.1,b0659ab26722f7b229688ccacfd108cb
Lung_Opacity-284.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-284.png,Lung_Opacity/images/Lung_Opacity-284.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-284.png,Lung_Opacity/masks/Lung_Opacity-284.png,True,False,299,299,L,137.16,3edfb3be7c3343e7d832793dcfa60f04
Lung_Opacity-2840.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2840.png,Lung_Opacity/images/Lung_Opacity-2840.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2840.png,Lung_Opacity/masks/Lung_Opacity-2840.png,True,False,299,299,L,115.95,940d141c4626c8d000c14e51578de277
Lung_Opacity-2841.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2841.png,Lung_Opacity/images/Lung_Opacity-2841.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2841.png,Lung_Opacity/masks/Lung_Opacity-2841.png,True,False,299,299,L,99.76,9a5dba1dde6b9b53a918ccb2a0e9d46c
Lung_Opacity-2842.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2842.png,Lung_Opacity/images/Lung_Opacity-2842.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2842.png,Lung_Opacity/masks/Lung_Opacity-2842.png,True,False,299,299,L,133.73,28ec2654d3e72a6e8d8f7ae0bf9402de
Lung_Opacity-2843.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2843.png,Lung_Opacity/images/Lung_Opacity-2843.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2843.png,Lung_Opacity/masks/Lung_Opacity-2843.png,True,False,299,299,L,110.93,004974ee0589ab8888028fbd844a4030
Lung_Opacity-2844.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2844.png,Lung_Opacity/images/Lung_Opacity-2844.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2844.png,Lung_Opacity/masks/Lung_Opacity-2844.png,True,False,299,299,L,127.94,c5496da565c8b0a326258d3d0b28d0d8
Lung_Opacity-2845.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2845.png,Lung_Opacity/images/Lung_Opacity-2845.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2845.png,Lung_Opacity/masks/Lung_Opacity-2845.png,True,False,299,299,L,163.1,5ffd23353ef5224d0552d31f82b72a33
Lung_Opacity-2846.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2846.png,Lung_Opacity/images/Lung_Opacity-2846.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2846.png,Lung_Opacity/masks/Lung_Opacity-2846.png,True,False,299,299,L,160.1,9c4616d2dab5b8bdbfe660a9ca02d284
Lung_Opacity-2847.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2847.png,Lung_Opacity/images/Lung_Opacity-2847.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2847.png,Lung_Opacity/masks/Lung_Opacity-2847.png,True,False,299,299,L,104.88,6725f0cd893d595c9520a8ef2f4202f6
Lung_Opacity-2848.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2848.png,Lung_Opacity/images/Lung_Opacity-2848.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2848.png,Lung_Opacity/masks/Lung_Opacity-2848.png,True,False,299,299,L,104.04,5c72700dbaed66139b5e44b97d128b3a
Lung_Opacity-2849.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2849.png,Lung_Opacity/images/Lung_Opacity-2849.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2849.png,Lung_Opacity/masks/Lung_Opacity-2849.png,True,False,299,299,L,147.57,ce9863ad5543c443e20d3bc7a8dbd3a6
Lung_Opacity-285.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-285.png,Lung_Opacity/images/Lung_Opacity-285.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-285.png,Lung_Opacity/masks/Lung_Opacity-285.png,True,False,299,299,L,118.9,7a1a326312b74c498cc951a0f2354464
Lung_Opacity-2850.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2850.png,Lung_Opacity/images/Lung_Opacity-2850.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2850.png,Lung_Opacity/masks/Lung_Opacity-2850.png,True,False,299,299,L,130.8,bf9b4e16cca3c6bff8910468fdad8da8
Lung_Opacity-2851.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2851.png,Lung_Opacity/images/Lung_Opacity-2851.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2851.png,Lung_Opacity/masks/Lung_Opacity-2851.png,True,False,299,299,L,162.61,7e74a74635c650817f1b53139c090a7a
Lung_Opacity-2852.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2852.png,Lung_Opacity/images/Lung_Opacity-2852.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2852.png,Lung_Opacity/masks/Lung_Opacity-2852.png,True,False,299,299,L,120.23,8d80d7a0bf749a71a59a45149b336449
Lung_Opacity-2853.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2853.png,Lung_Opacity/images/Lung_Opacity-2853.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2853.png,Lung_Opacity/masks/Lung_Opacity-2853.png,True,False,299,299,L,94.94,6a418fced14f95bcce96f91d4655988b
Lung_Opacity-2854.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2854.png,Lung_Opacity/images/Lung_Opacity-2854.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2854.png,Lung_Opacity/masks/Lung_Opacity-2854.png,True,False,299,299,L,127.11,3313cc0fc9e234d1941a48440bad7f14
Lung_Opacity-2855.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2855.png,Lung_Opacity/images/Lung_Opacity-2855.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2855.png,Lung_Opacity/masks/Lung_Opacity-2855.png,True,False,299,299,L,117.18,ceddf20e3ebe9ce139a8c7cb114b1ea8
Lung_Opacity-2856.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2856.png,Lung_Opacity/images/Lung_Opacity-2856.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2856.png,Lung_Opacity/masks/Lung_Opacity-2856.png,True,False,299,299,L,116.85,98d14febee433993eb706e16f1380388
Lung_Opacity-2857.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2857.png,Lung_Opacity/images/Lung_Opacity-2857.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2857.png,Lung_Opacity/masks/Lung_Opacity-2857.png,True,False,299,299,L,106.7,81116244943f2f8922d918e686048476
Lung_Opacity-2858.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2858.png,Lung_Opacity/images/Lung_Opacity-2858.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2858.png,Lung_Opacity/masks/Lung_Opacity-2858.png,True,False,299,299,L,120.4,0213a2592848fb5d9f57e65a9f01f111
Lung_Opacity-2859.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2859.png,Lung_Opacity/images/Lung_Opacity-2859.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2859.png,Lung_Opacity/masks/Lung_Opacity-2859.png,True,False,299,299,L,129.93,bca0c795eb38126a67d5e38ff67fc6af
Lung_Opacity-286.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-286.png,Lung_Opacity/images/Lung_Opacity-286.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-286.png,Lung_Opacity/masks/Lung_Opacity-286.png,True,False,299,299,L,88.46,655a167fd9b25a98360db36789236c86
Lung_Opacity-2860.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2860.png,Lung_Opacity/images/Lung_Opacity-2860.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2860.png,Lung_Opacity/masks/Lung_Opacity-2860.png,True,False,299,299,L,121.39,feead7b8ad71abc24f05c5c3cbfece8a
Lung_Opacity-2861.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2861.png,Lung_Opacity/images/Lung_Opacity-2861.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2861.png,Lung_Opacity/masks/Lung_Opacity-2861.png,True,False,299,299,L,119.74,11acb89a9eb2f5f3c90dff8a2ea407bb
Lung_Opacity-2862.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2862.png,Lung_Opacity/images/Lung_Opacity-2862.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2862.png,Lung_Opacity/masks/Lung_Opacity-2862.png,True,False,299,299,L,121.93,e94674a4f86ded9a46b387c5188973aa
Lung_Opacity-2863.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2863.png,Lung_Opacity/images/Lung_Opacity-2863.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2863.png,Lung_Opacity/masks/Lung_Opacity-2863.png,True,False,299,299,L,130.32,352b699ab0de7fe4b793e18a3ba883b1
Lung_Opacity-2864.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2864.png,Lung_Opacity/images/Lung_Opacity-2864.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2864.png,Lung_Opacity/masks/Lung_Opacity-2864.png,True,False,299,299,L,117.17,faad61cecfca140d30cc02376b8356e2
Lung_Opacity-2865.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2865.png,Lung_Opacity/images/Lung_Opacity-2865.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2865.png,Lung_Opacity/masks/Lung_Opacity-2865.png,True,False,299,299,L,161.21,ad69eec01d95ad6fefc2097af943a000
Lung_Opacity-2866.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2866.png,Lung_Opacity/images/Lung_Opacity-2866.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2866.png,Lung_Opacity/masks/Lung_Opacity-2866.png,True,False,299,299,L,118.66,4c94a6176da59de5d808b3d9ebdbafe4
Lung_Opacity-2867.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2867.png,Lung_Opacity/images/Lung_Opacity-2867.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2867.png,Lung_Opacity/masks/Lung_Opacity-2867.png,True,False,299,299,L,110.77,444414f6139dc091df685b81a97fbc1f
Lung_Opacity-2868.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2868.png,Lung_Opacity/images/Lung_Opacity-2868.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2868.png,Lung_Opacity/masks/Lung_Opacity-2868.png,True,False,299,299,L,139.08,e79ad8cf7ad7c6221d854d7cacd25456
Lung_Opacity-2869.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2869.png,Lung_Opacity/images/Lung_Opacity-2869.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2869.png,Lung_Opacity/masks/Lung_Opacity-2869.png,True,False,299,299,L,129.84,ee79b33ef528b602adfac138968806e2
Lung_Opacity-287.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-287.png,Lung_Opacity/images/Lung_Opacity-287.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-287.png,Lung_Opacity/masks/Lung_Opacity-287.png,True,False,299,299,L,110.66,4a9769180a5c7239046b9dfb61aee0b6
Lung_Opacity-2870.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2870.png,Lung_Opacity/images/Lung_Opacity-2870.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2870.png,Lung_Opacity/masks/Lung_Opacity-2870.png,True,False,299,299,L,111.0,942bd312c5866098361946083f1b1acb
Lung_Opacity-2871.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2871.png,Lung_Opacity/images/Lung_Opacity-2871.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2871.png,Lung_Opacity/masks/Lung_Opacity-2871.png,True,False,299,299,L,131.96,a8e39bb4ea354a1319da0376a956425b
Lung_Opacity-2872.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2872.png,Lung_Opacity/images/Lung_Opacity-2872.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2872.png,Lung_Opacity/masks/Lung_Opacity-2872.png,True,False,299,299,L,105.22,c4e389c760e77eb73656eab540b85c82
Lung_Opacity-2873.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2873.png,Lung_Opacity/images/Lung_Opacity-2873.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2873.png,Lung_Opacity/masks/Lung_Opacity-2873.png,True,False,299,299,L,167.09,836639c8fb05e44d5ec95312d20fddf3
Lung_Opacity-2874.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2874.png,Lung_Opacity/images/Lung_Opacity-2874.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2874.png,Lung_Opacity/masks/Lung_Opacity-2874.png,True,False,299,299,L,162.99,c875dc363d37022b3ab1d75be7546e80
Lung_Opacity-2875.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2875.png,Lung_Opacity/images/Lung_Opacity-2875.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2875.png,Lung_Opacity/masks/Lung_Opacity-2875.png,True,False,299,299,L,128.45,a936810ef02ebee242d41882bccc9e39
Lung_Opacity-2876.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2876.png,Lung_Opacity/images/Lung_Opacity-2876.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2876.png,Lung_Opacity/masks/Lung_Opacity-2876.png,True,False,299,299,L,142.22,60ed853067f2718aa9ea66a6cdceeed0
Lung_Opacity-2877.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2877.png,Lung_Opacity/images/Lung_Opacity-2877.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2877.png,Lung_Opacity/masks/Lung_Opacity-2877.png,True,False,299,299,L,131.06,5a876032b701fdb44e0af41f4853c148
Lung_Opacity-2878.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2878.png,Lung_Opacity/images/Lung_Opacity-2878.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2878.png,Lung_Opacity/masks/Lung_Opacity-2878.png,True,False,299,299,L,157.75,511802769823b2776f60f08b6478789c
Lung_Opacity-2879.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2879.png,Lung_Opacity/images/Lung_Opacity-2879.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2879.png,Lung_Opacity/masks/Lung_Opacity-2879.png,True,False,299,299,L,177.39,83eac42b8ea20d181cdfb97f8f234003
Lung_Opacity-288.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-288.png,Lung_Opacity/images/Lung_Opacity-288.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-288.png,Lung_Opacity/masks/Lung_Opacity-288.png,True,False,299,299,L,163.07,61569cef9dfd13852044ec40852e6644
Lung_Opacity-2880.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2880.png,Lung_Opacity/images/Lung_Opacity-2880.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2880.png,Lung_Opacity/masks/Lung_Opacity-2880.png,True,False,299,299,L,160.53,6829019c2b24f8e9641ecd3146283346
Lung_Opacity-2881.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2881.png,Lung_Opacity/images/Lung_Opacity-2881.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2881.png,Lung_Opacity/masks/Lung_Opacity-2881.png,True,False,299,299,L,114.3,f6c0cf2d2caa9e248ddfe9abb9d3ae6e
Lung_Opacity-2882.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2882.png,Lung_Opacity/images/Lung_Opacity-2882.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2882.png,Lung_Opacity/masks/Lung_Opacity-2882.png,True,False,299,299,L,103.34,7ac5fb19c4e5424bdc559e268f5b8de0
Lung_Opacity-2883.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2883.png,Lung_Opacity/images/Lung_Opacity-2883.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2883.png,Lung_Opacity/masks/Lung_Opacity-2883.png,True,False,299,299,L,90.36,cc4107c7c8bc1951a22a1ac7153e49f3
Lung_Opacity-2884.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2884.png,Lung_Opacity/images/Lung_Opacity-2884.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2884.png,Lung_Opacity/masks/Lung_Opacity-2884.png,True,False,299,299,L,166.39,314870f0523fecb61c982a1f85d2f6dd
Lung_Opacity-2885.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2885.png,Lung_Opacity/images/Lung_Opacity-2885.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2885.png,Lung_Opacity/masks/Lung_Opacity-2885.png,True,False,299,299,L,118.01,e3df685ee24b1b956b44b8f10766520b
Lung_Opacity-2886.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2886.png,Lung_Opacity/images/Lung_Opacity-2886.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2886.png,Lung_Opacity/masks/Lung_Opacity-2886.png,True,False,299,299,L,99.03,f340d3798d946288e8d0c2578c9a68cb
Lung_Opacity-2887.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2887.png,Lung_Opacity/images/Lung_Opacity-2887.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2887.png,Lung_Opacity/masks/Lung_Opacity-2887.png,True,False,299,299,L,103.68,fbbc1c2e93947600c4f9909d13f7f817
Lung_Opacity-2888.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2888.png,Lung_Opacity/images/Lung_Opacity-2888.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2888.png,Lung_Opacity/masks/Lung_Opacity-2888.png,True,False,299,299,L,106.25,ffb4e8282cb2bdcbdb848bb29391afe5
Lung_Opacity-2889.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2889.png,Lung_Opacity/images/Lung_Opacity-2889.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2889.png,Lung_Opacity/masks/Lung_Opacity-2889.png,True,False,299,299,L,113.11,298a42d88ad7759e3ca4c23217011160
Lung_Opacity-289.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-289.png,Lung_Opacity/images/Lung_Opacity-289.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-289.png,Lung_Opacity/masks/Lung_Opacity-289.png,True,False,299,299,L,142.68,541762548a35cec94e2fbfec7a13f746
Lung_Opacity-2890.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2890.png,Lung_Opacity/images/Lung_Opacity-2890.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2890.png,Lung_Opacity/masks/Lung_Opacity-2890.png,True,False,299,299,L,133.83,2e1e917e77501e4eb8ec4cd8782010f2
Lung_Opacity-2891.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2891.png,Lung_Opacity/images/Lung_Opacity-2891.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2891.png,Lung_Opacity/masks/Lung_Opacity-2891.png,True,False,299,299,L,109.65,c08b0a3c7ba2b2d66db7e377f336be94
Lung_Opacity-2892.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2892.png,Lung_Opacity/images/Lung_Opacity-2892.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2892.png,Lung_Opacity/masks/Lung_Opacity-2892.png,True,False,299,299,L,107.21,adb5601acbab89f2aba48e77b9ce805d
Lung_Opacity-2893.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2893.png,Lung_Opacity/images/Lung_Opacity-2893.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2893.png,Lung_Opacity/masks/Lung_Opacity-2893.png,True,False,299,299,L,105.97,8e58c950d47a12f540f4ff0f696cd77d
Lung_Opacity-2894.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2894.png,Lung_Opacity/images/Lung_Opacity-2894.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2894.png,Lung_Opacity/masks/Lung_Opacity-2894.png,True,False,299,299,L,153.42,9fec18f891a9737d84ec102f52db4b15
Lung_Opacity-2895.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2895.png,Lung_Opacity/images/Lung_Opacity-2895.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2895.png,Lung_Opacity/masks/Lung_Opacity-2895.png,True,False,299,299,L,159.86,4552dc13d10b970535fedff5e9c58795
Lung_Opacity-2896.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2896.png,Lung_Opacity/images/Lung_Opacity-2896.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2896.png,Lung_Opacity/masks/Lung_Opacity-2896.png,True,False,299,299,L,110.02,5f01783b8e1ef5f52466a04fd367698f
Lung_Opacity-2897.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2897.png,Lung_Opacity/images/Lung_Opacity-2897.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2897.png,Lung_Opacity/masks/Lung_Opacity-2897.png,True,False,299,299,L,100.67,e9be84a4a953001df386ff790b8459e4
Lung_Opacity-2898.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2898.png,Lung_Opacity/images/Lung_Opacity-2898.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2898.png,Lung_Opacity/masks/Lung_Opacity-2898.png,True,False,299,299,L,104.7,e3c4208292c56f9cbf6f1b30339cabed
Lung_Opacity-2899.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2899.png,Lung_Opacity/images/Lung_Opacity-2899.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2899.png,Lung_Opacity/masks/Lung_Opacity-2899.png,True,False,299,299,L,122.77,b5da1a257188f6d2c2d4a212fd043f03
Lung_Opacity-29.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-29.png,Lung_Opacity/images/Lung_Opacity-29.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-29.png,Lung_Opacity/masks/Lung_Opacity-29.png,True,False,299,299,L,108.66,7890fe0f4035b15fc65c0edd106a462b
Lung_Opacity-290.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-290.png,Lung_Opacity/images/Lung_Opacity-290.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-290.png,Lung_Opacity/masks/Lung_Opacity-290.png,True,False,299,299,L,129.57,57204ccef0abb6069228bd25b25b7474
Lung_Opacity-2900.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2900.png,Lung_Opacity/images/Lung_Opacity-2900.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2900.png,Lung_Opacity/masks/Lung_Opacity-2900.png,True,False,299,299,L,119.33,acbab4410d7f8e60c604f19657af7ace
Lung_Opacity-2901.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2901.png,Lung_Opacity/images/Lung_Opacity-2901.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2901.png,Lung_Opacity/masks/Lung_Opacity-2901.png,True,False,299,299,L,111.28,9a7f08a01849b5215959eb70e6e64348
Lung_Opacity-2902.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2902.png,Lung_Opacity/images/Lung_Opacity-2902.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2902.png,Lung_Opacity/masks/Lung_Opacity-2902.png,True,False,299,299,L,114.2,e50b27e728ad299a069bf0e709d51c34
Lung_Opacity-2903.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2903.png,Lung_Opacity/images/Lung_Opacity-2903.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2903.png,Lung_Opacity/masks/Lung_Opacity-2903.png,True,False,299,299,L,108.24,77564819421134da059c7f03d9065681
Lung_Opacity-2904.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2904.png,Lung_Opacity/images/Lung_Opacity-2904.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2904.png,Lung_Opacity/masks/Lung_Opacity-2904.png,True,False,299,299,L,95.93,6c6a0f36169c6fd9c3695abb20934f91
Lung_Opacity-2905.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2905.png,Lung_Opacity/images/Lung_Opacity-2905.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2905.png,Lung_Opacity/masks/Lung_Opacity-2905.png,True,False,299,299,L,122.93,2468faa5b81c32d3fe39a71f92cd907b
Lung_Opacity-2906.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2906.png,Lung_Opacity/images/Lung_Opacity-2906.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2906.png,Lung_Opacity/masks/Lung_Opacity-2906.png,True,False,299,299,L,147.65,3a2924a65b46e4f20339b33ff358dfb9
Lung_Opacity-2907.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2907.png,Lung_Opacity/images/Lung_Opacity-2907.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2907.png,Lung_Opacity/masks/Lung_Opacity-2907.png,True,False,299,299,L,171.93,4b48d048044516cc3455e28d7fad0927
Lung_Opacity-2908.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2908.png,Lung_Opacity/images/Lung_Opacity-2908.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2908.png,Lung_Opacity/masks/Lung_Opacity-2908.png,True,False,299,299,L,170.81,e059b70e9818aa18ddf47beb290315bb
Lung_Opacity-2909.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2909.png,Lung_Opacity/images/Lung_Opacity-2909.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2909.png,Lung_Opacity/masks/Lung_Opacity-2909.png,True,False,299,299,L,116.76,4a248da8c15d79f96ccb939e6b60ace2
Lung_Opacity-291.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-291.png,Lung_Opacity/images/Lung_Opacity-291.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-291.png,Lung_Opacity/masks/Lung_Opacity-291.png,True,False,299,299,L,102.98,4a846c847c077d042c9a57e5a7f264f8
Lung_Opacity-2910.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2910.png,Lung_Opacity/images/Lung_Opacity-2910.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2910.png,Lung_Opacity/masks/Lung_Opacity-2910.png,True,False,299,299,L,122.93,2af38bb0ebf08bc7da2a0193384f0c45
Lung_Opacity-2911.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2911.png,Lung_Opacity/images/Lung_Opacity-2911.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2911.png,Lung_Opacity/masks/Lung_Opacity-2911.png,True,False,299,299,L,123.03,96663a3bd3b8984800a8923d5eaf62a7
Lung_Opacity-2912.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2912.png,Lung_Opacity/images/Lung_Opacity-2912.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2912.png,Lung_Opacity/masks/Lung_Opacity-2912.png,True,False,299,299,L,137.62,0dd0ce33c1e5200e4b8f227715eb52c7
Lung_Opacity-2913.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2913.png,Lung_Opacity/images/Lung_Opacity-2913.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2913.png,Lung_Opacity/masks/Lung_Opacity-2913.png,True,False,299,299,L,89.41,f4be7ecac93053720b93546030648c2c
Lung_Opacity-2914.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2914.png,Lung_Opacity/images/Lung_Opacity-2914.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2914.png,Lung_Opacity/masks/Lung_Opacity-2914.png,True,False,299,299,L,104.15,b3ba56b099e9e9cdf84c71f728c9ab51
Lung_Opacity-2915.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2915.png,Lung_Opacity/images/Lung_Opacity-2915.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2915.png,Lung_Opacity/masks/Lung_Opacity-2915.png,True,False,299,299,L,128.97,bf0920223d06ac3c0aa7db3bab31d1d3
Lung_Opacity-2916.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2916.png,Lung_Opacity/images/Lung_Opacity-2916.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2916.png,Lung_Opacity/masks/Lung_Opacity-2916.png,True,False,299,299,L,157.54,ffcb099136921db1de974d33467f39b0
Lung_Opacity-2917.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2917.png,Lung_Opacity/images/Lung_Opacity-2917.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2917.png,Lung_Opacity/masks/Lung_Opacity-2917.png,True,False,299,299,L,111.14,9c9c38c8fbf8e0d7843d5b641cc2425d
Lung_Opacity-2918.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2918.png,Lung_Opacity/images/Lung_Opacity-2918.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2918.png,Lung_Opacity/masks/Lung_Opacity-2918.png,True,False,299,299,L,101.98,ac5f55ef0127c092ec22724073e59a71
Lung_Opacity-2919.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2919.png,Lung_Opacity/images/Lung_Opacity-2919.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2919.png,Lung_Opacity/masks/Lung_Opacity-2919.png,True,False,299,299,L,124.41,2f3e11f4c76cc590cb4d0ec73a547dd5
Lung_Opacity-292.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-292.png,Lung_Opacity/images/Lung_Opacity-292.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-292.png,Lung_Opacity/masks/Lung_Opacity-292.png,True,False,299,299,L,149.74,fc6e9f5bde1e0463d1bf13fea22d5985
Lung_Opacity-2920.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2920.png,Lung_Opacity/images/Lung_Opacity-2920.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2920.png,Lung_Opacity/masks/Lung_Opacity-2920.png,True,False,299,299,L,114.97,93249ff72be530181a9cdabcb431d5e8
Lung_Opacity-2921.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2921.png,Lung_Opacity/images/Lung_Opacity-2921.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2921.png,Lung_Opacity/masks/Lung_Opacity-2921.png,True,False,299,299,L,120.59,79bbfee381abe711113110a9fa31434b
Lung_Opacity-2922.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2922.png,Lung_Opacity/images/Lung_Opacity-2922.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2922.png,Lung_Opacity/masks/Lung_Opacity-2922.png,True,False,299,299,L,175.83,c9fd0be5df9fd1e9a1cf4cde8c893aba
Lung_Opacity-2923.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2923.png,Lung_Opacity/images/Lung_Opacity-2923.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2923.png,Lung_Opacity/masks/Lung_Opacity-2923.png,True,False,299,299,L,63.8,566499ef223cf9891db51ce7b1067281
Lung_Opacity-2924.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2924.png,Lung_Opacity/images/Lung_Opacity-2924.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2924.png,Lung_Opacity/masks/Lung_Opacity-2924.png,True,False,299,299,L,108.8,3aaf3d313fcbbdebe8c8b7064245ed3d
Lung_Opacity-2925.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2925.png,Lung_Opacity/images/Lung_Opacity-2925.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2925.png,Lung_Opacity/masks/Lung_Opacity-2925.png,True,False,299,299,L,124.6,ec7364b73e4f343347b532d3e9631d9d
Lung_Opacity-2926.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2926.png,Lung_Opacity/images/Lung_Opacity-2926.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2926.png,Lung_Opacity/masks/Lung_Opacity-2926.png,True,False,299,299,L,115.52,542129bdeecddb6409f8d3eefdaa5dde
Lung_Opacity-2927.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2927.png,Lung_Opacity/images/Lung_Opacity-2927.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2927.png,Lung_Opacity/masks/Lung_Opacity-2927.png,True,False,299,299,L,120.8,4fa9067c20772927b2ab25b5397b0807
Lung_Opacity-2928.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2928.png,Lung_Opacity/images/Lung_Opacity-2928.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2928.png,Lung_Opacity/masks/Lung_Opacity-2928.png,True,False,299,299,L,147.92,24c6f9eb905aefc1459c03b3cc11cc1c
Lung_Opacity-2929.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2929.png,Lung_Opacity/images/Lung_Opacity-2929.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2929.png,Lung_Opacity/masks/Lung_Opacity-2929.png,True,False,299,299,L,112.27,992ead40f56a8543ed5c9cebfcc6db61
Lung_Opacity-293.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-293.png,Lung_Opacity/images/Lung_Opacity-293.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-293.png,Lung_Opacity/masks/Lung_Opacity-293.png,True,False,299,299,L,119.14,f2bbc0283d6e1c0fec51e63341e82e0d
Lung_Opacity-2930.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2930.png,Lung_Opacity/images/Lung_Opacity-2930.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2930.png,Lung_Opacity/masks/Lung_Opacity-2930.png,True,False,299,299,L,87.1,3b9f221813dc21bfb495561cc5c2240d
Lung_Opacity-2931.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2931.png,Lung_Opacity/images/Lung_Opacity-2931.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2931.png,Lung_Opacity/masks/Lung_Opacity-2931.png,True,False,299,299,L,105.41,ac8ea09d1b2edd25ffcef5c716baf53d
Lung_Opacity-2932.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2932.png,Lung_Opacity/images/Lung_Opacity-2932.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2932.png,Lung_Opacity/masks/Lung_Opacity-2932.png,True,False,299,299,L,116.96,70bf594d9301a6b61bd858a082ff3ed9
Lung_Opacity-2933.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2933.png,Lung_Opacity/images/Lung_Opacity-2933.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2933.png,Lung_Opacity/masks/Lung_Opacity-2933.png,True,False,299,299,L,123.67,1d7486a5bf571d823f1fbdf705eeb9c9
Lung_Opacity-2934.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2934.png,Lung_Opacity/images/Lung_Opacity-2934.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2934.png,Lung_Opacity/masks/Lung_Opacity-2934.png,True,False,299,299,L,119.95,065970b9792e5e86676f5c7c41da7419
Lung_Opacity-2935.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2935.png,Lung_Opacity/images/Lung_Opacity-2935.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2935.png,Lung_Opacity/masks/Lung_Opacity-2935.png,True,False,299,299,L,124.48,d603aa53be6b2ff2d1c40648986569d0
Lung_Opacity-2936.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2936.png,Lung_Opacity/images/Lung_Opacity-2936.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2936.png,Lung_Opacity/masks/Lung_Opacity-2936.png,True,False,299,299,L,115.69,88a2d2d0f8b4a4e256bbaa7e7ffcb74a
Lung_Opacity-2937.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2937.png,Lung_Opacity/images/Lung_Opacity-2937.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2937.png,Lung_Opacity/masks/Lung_Opacity-2937.png,True,False,299,299,L,112.19,483ca83f209b49f730b2dbb54898fb63
Lung_Opacity-2938.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2938.png,Lung_Opacity/images/Lung_Opacity-2938.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2938.png,Lung_Opacity/masks/Lung_Opacity-2938.png,True,False,299,299,L,125.42,255c7953562fa4f93b64449464e9ae29
Lung_Opacity-2939.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2939.png,Lung_Opacity/images/Lung_Opacity-2939.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2939.png,Lung_Opacity/masks/Lung_Opacity-2939.png,True,False,299,299,L,120.16,4d4c742fe4c7a0801738232522bae5cb
Lung_Opacity-294.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-294.png,Lung_Opacity/images/Lung_Opacity-294.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-294.png,Lung_Opacity/masks/Lung_Opacity-294.png,True,False,299,299,L,115.43,b59304065a1ff0937b66bad325b7e153
Lung_Opacity-2940.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2940.png,Lung_Opacity/images/Lung_Opacity-2940.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2940.png,Lung_Opacity/masks/Lung_Opacity-2940.png,True,False,299,299,L,99.74,41788a7a900225573b293f4d430583b4
Lung_Opacity-2941.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2941.png,Lung_Opacity/images/Lung_Opacity-2941.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2941.png,Lung_Opacity/masks/Lung_Opacity-2941.png,True,False,299,299,L,99.55,6c2e708f2aaf8c9bfe7efac92512143f
Lung_Opacity-2942.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2942.png,Lung_Opacity/images/Lung_Opacity-2942.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2942.png,Lung_Opacity/masks/Lung_Opacity-2942.png,True,False,299,299,L,110.56,10887741cb123d8067e9c3aa1945d87e
Lung_Opacity-2943.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2943.png,Lung_Opacity/images/Lung_Opacity-2943.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2943.png,Lung_Opacity/masks/Lung_Opacity-2943.png,True,False,299,299,L,111.8,fa9d93b7bea17e46d9a8e9dc451a4ed6
Lung_Opacity-2944.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2944.png,Lung_Opacity/images/Lung_Opacity-2944.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2944.png,Lung_Opacity/masks/Lung_Opacity-2944.png,True,False,299,299,L,115.2,6e2a5d4a71de813b405fd7d33ab6829d
Lung_Opacity-2945.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2945.png,Lung_Opacity/images/Lung_Opacity-2945.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2945.png,Lung_Opacity/masks/Lung_Opacity-2945.png,True,False,299,299,L,156.46,cba09ab5b91d1ef3c8fd69668c215503
Lung_Opacity-2946.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2946.png,Lung_Opacity/images/Lung_Opacity-2946.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2946.png,Lung_Opacity/masks/Lung_Opacity-2946.png,True,False,299,299,L,119.19,e890feffbd3a1ff496ca1f0a258322e7
Lung_Opacity-2947.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2947.png,Lung_Opacity/images/Lung_Opacity-2947.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2947.png,Lung_Opacity/masks/Lung_Opacity-2947.png,True,False,299,299,L,113.93,bc11c7db0b9d228e121eb16c460fd4a4
Lung_Opacity-2948.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2948.png,Lung_Opacity/images/Lung_Opacity-2948.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2948.png,Lung_Opacity/masks/Lung_Opacity-2948.png,True,False,299,299,L,94.87,412a48955aa1bdd9150a62109483287e
Lung_Opacity-2949.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2949.png,Lung_Opacity/images/Lung_Opacity-2949.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2949.png,Lung_Opacity/masks/Lung_Opacity-2949.png,True,False,299,299,L,134.13,652eb68ec9df2159a25377423b868c8b
Lung_Opacity-295.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-295.png,Lung_Opacity/images/Lung_Opacity-295.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-295.png,Lung_Opacity/masks/Lung_Opacity-295.png,True,False,299,299,L,147.83,c9de4add76d967d2544a33b6fbd704c8
Lung_Opacity-2950.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2950.png,Lung_Opacity/images/Lung_Opacity-2950.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2950.png,Lung_Opacity/masks/Lung_Opacity-2950.png,True,False,299,299,L,112.4,8b514b8772fa8df5b9438b238c302bbe
Lung_Opacity-2951.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2951.png,Lung_Opacity/images/Lung_Opacity-2951.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2951.png,Lung_Opacity/masks/Lung_Opacity-2951.png,True,False,299,299,L,119.48,a1756a62c432ac7d1090e31778d493f3
Lung_Opacity-2952.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2952.png,Lung_Opacity/images/Lung_Opacity-2952.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2952.png,Lung_Opacity/masks/Lung_Opacity-2952.png,True,False,299,299,L,114.31,b21e9e13e9a313a631c8cf4a2f5d50a7
Lung_Opacity-2953.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2953.png,Lung_Opacity/images/Lung_Opacity-2953.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2953.png,Lung_Opacity/masks/Lung_Opacity-2953.png,True,False,299,299,L,125.23,e531dc8ed9aa58f8d9533b6a6c712305
Lung_Opacity-2954.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2954.png,Lung_Opacity/images/Lung_Opacity-2954.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2954.png,Lung_Opacity/masks/Lung_Opacity-2954.png,True,False,299,299,L,141.67,1d8f5af0ee44bfa449758789241ecda3
Lung_Opacity-2955.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2955.png,Lung_Opacity/images/Lung_Opacity-2955.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2955.png,Lung_Opacity/masks/Lung_Opacity-2955.png,True,False,299,299,L,106.46,ef1bc61ec3245b5591178effcb85fd95
Lung_Opacity-2956.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2956.png,Lung_Opacity/images/Lung_Opacity-2956.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2956.png,Lung_Opacity/masks/Lung_Opacity-2956.png,True,False,299,299,L,164.28,18b971e8d977788eb29733a69d409444
Lung_Opacity-2957.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2957.png,Lung_Opacity/images/Lung_Opacity-2957.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2957.png,Lung_Opacity/masks/Lung_Opacity-2957.png,True,False,299,299,L,92.55,8ce0119a99485692af1ca68b7d3de57a
Lung_Opacity-2958.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2958.png,Lung_Opacity/images/Lung_Opacity-2958.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2958.png,Lung_Opacity/masks/Lung_Opacity-2958.png,True,False,299,299,L,126.12,7539dbc6a931c22bacc63535c1080dab
Lung_Opacity-2959.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2959.png,Lung_Opacity/images/Lung_Opacity-2959.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2959.png,Lung_Opacity/masks/Lung_Opacity-2959.png,True,False,299,299,L,120.58,8ef3a7ce614aecd31f4011bfd539cab1
Lung_Opacity-296.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-296.png,Lung_Opacity/images/Lung_Opacity-296.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-296.png,Lung_Opacity/masks/Lung_Opacity-296.png,True,False,299,299,L,106.56,568c78135bcaa20fe6d006c2238c12ef
Lung_Opacity-2960.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2960.png,Lung_Opacity/images/Lung_Opacity-2960.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2960.png,Lung_Opacity/masks/Lung_Opacity-2960.png,True,False,299,299,L,124.66,01fe2f35fd0a9c83444f4b7ddf994723
Lung_Opacity-2961.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2961.png,Lung_Opacity/images/Lung_Opacity-2961.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2961.png,Lung_Opacity/masks/Lung_Opacity-2961.png,True,False,299,299,L,132.82,07ad656f34e997e6f8edcee6ea725603
Lung_Opacity-2962.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2962.png,Lung_Opacity/images/Lung_Opacity-2962.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2962.png,Lung_Opacity/masks/Lung_Opacity-2962.png,True,False,299,299,L,155.09,16be8b3edd9f77fde9054eb6fa039816
Lung_Opacity-2963.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2963.png,Lung_Opacity/images/Lung_Opacity-2963.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2963.png,Lung_Opacity/masks/Lung_Opacity-2963.png,True,False,299,299,L,110.68,8d8e37a69734df18f76cc20f43720f36
Lung_Opacity-2964.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2964.png,Lung_Opacity/images/Lung_Opacity-2964.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2964.png,Lung_Opacity/masks/Lung_Opacity-2964.png,True,False,299,299,L,160.62,ecca8d4f9d53105fdf598f252bda4805
Lung_Opacity-2965.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2965.png,Lung_Opacity/images/Lung_Opacity-2965.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2965.png,Lung_Opacity/masks/Lung_Opacity-2965.png,True,False,299,299,L,142.78,1d1358e941398173839ca8dbcaedd965
Lung_Opacity-2966.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2966.png,Lung_Opacity/images/Lung_Opacity-2966.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2966.png,Lung_Opacity/masks/Lung_Opacity-2966.png,True,False,299,299,L,147.36,c69ef2ed1436bd241566bcd8aa00b3b9
Lung_Opacity-2967.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2967.png,Lung_Opacity/images/Lung_Opacity-2967.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2967.png,Lung_Opacity/masks/Lung_Opacity-2967.png,True,False,299,299,L,114.78,58346629132e655e41a7de35e9f240f6
Lung_Opacity-2968.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2968.png,Lung_Opacity/images/Lung_Opacity-2968.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2968.png,Lung_Opacity/masks/Lung_Opacity-2968.png,True,False,299,299,L,158.28,9e636bf40769828beb9e7d5d73748129
Lung_Opacity-2969.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2969.png,Lung_Opacity/images/Lung_Opacity-2969.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2969.png,Lung_Opacity/masks/Lung_Opacity-2969.png,True,False,299,299,L,158.06,282b8a76597f5c63c68de84da85585ef
Lung_Opacity-297.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-297.png,Lung_Opacity/images/Lung_Opacity-297.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-297.png,Lung_Opacity/masks/Lung_Opacity-297.png,True,False,299,299,L,114.22,6d3753e6fd4263fc82391fff933602b8
Lung_Opacity-2970.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2970.png,Lung_Opacity/images/Lung_Opacity-2970.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2970.png,Lung_Opacity/masks/Lung_Opacity-2970.png,True,False,299,299,L,160.94,723716879e71792e7bf1a075c5ff968e
Lung_Opacity-2971.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2971.png,Lung_Opacity/images/Lung_Opacity-2971.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2971.png,Lung_Opacity/masks/Lung_Opacity-2971.png,True,False,299,299,L,162.97,7881b2450fdb00ed9032dfba63bb2ea5
Lung_Opacity-2972.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2972.png,Lung_Opacity/images/Lung_Opacity-2972.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2972.png,Lung_Opacity/masks/Lung_Opacity-2972.png,True,False,299,299,L,149.23,0e4fd14b83c232f5e8f5b7b339633d2f
Lung_Opacity-2973.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2973.png,Lung_Opacity/images/Lung_Opacity-2973.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2973.png,Lung_Opacity/masks/Lung_Opacity-2973.png,True,False,299,299,L,127.93,4a5a2e9ecd6ba58784f9031ba8d190c6
Lung_Opacity-2974.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2974.png,Lung_Opacity/images/Lung_Opacity-2974.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2974.png,Lung_Opacity/masks/Lung_Opacity-2974.png,True,False,299,299,L,120.62,0c7785eb6b191d246003f0973d54b5c9
Lung_Opacity-2975.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2975.png,Lung_Opacity/images/Lung_Opacity-2975.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2975.png,Lung_Opacity/masks/Lung_Opacity-2975.png,True,False,299,299,L,147.58,03ca8f3ad0a3d2fbecb43fe0034b63e8
Lung_Opacity-2976.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2976.png,Lung_Opacity/images/Lung_Opacity-2976.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2976.png,Lung_Opacity/masks/Lung_Opacity-2976.png,True,False,299,299,L,125.22,78b89a3f28f3f6dafb5d0f3f0ada4d61
Lung_Opacity-2977.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2977.png,Lung_Opacity/images/Lung_Opacity-2977.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2977.png,Lung_Opacity/masks/Lung_Opacity-2977.png,True,False,299,299,L,104.75,429d3712d43f379e4bd0f0e10576d082
Lung_Opacity-2978.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2978.png,Lung_Opacity/images/Lung_Opacity-2978.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2978.png,Lung_Opacity/masks/Lung_Opacity-2978.png,True,False,299,299,L,112.29,607b13e103935457955814ed34646bc6
Lung_Opacity-2979.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2979.png,Lung_Opacity/images/Lung_Opacity-2979.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2979.png,Lung_Opacity/masks/Lung_Opacity-2979.png,True,False,299,299,L,108.11,54066281b0207efb6e71778f84ee8569
Lung_Opacity-298.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-298.png,Lung_Opacity/images/Lung_Opacity-298.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-298.png,Lung_Opacity/masks/Lung_Opacity-298.png,True,False,299,299,L,107.01,c1ab49db3c0d4d8ee5f2959cac363688
Lung_Opacity-2980.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2980.png,Lung_Opacity/images/Lung_Opacity-2980.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2980.png,Lung_Opacity/masks/Lung_Opacity-2980.png,True,False,299,299,L,82.85,65c2a14f5cc6207836fedd9da2b3a3bc
Lung_Opacity-2981.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2981.png,Lung_Opacity/images/Lung_Opacity-2981.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2981.png,Lung_Opacity/masks/Lung_Opacity-2981.png,True,False,299,299,L,118.02,4d5b8dfcbf4953ec8d142221e3f66d08
Lung_Opacity-2982.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2982.png,Lung_Opacity/images/Lung_Opacity-2982.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2982.png,Lung_Opacity/masks/Lung_Opacity-2982.png,True,False,299,299,L,118.89,ee5ea70c8949fb3ada5d08aedfeef337
Lung_Opacity-2983.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2983.png,Lung_Opacity/images/Lung_Opacity-2983.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2983.png,Lung_Opacity/masks/Lung_Opacity-2983.png,True,False,299,299,L,164.64,4db15d5c9d92667f74b8fa78622c14b2
Lung_Opacity-2984.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2984.png,Lung_Opacity/images/Lung_Opacity-2984.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2984.png,Lung_Opacity/masks/Lung_Opacity-2984.png,True,False,299,299,L,175.26,3f66b341851f279631a222ea62ecceae
Lung_Opacity-2985.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2985.png,Lung_Opacity/images/Lung_Opacity-2985.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2985.png,Lung_Opacity/masks/Lung_Opacity-2985.png,True,False,299,299,L,111.42,d50371dc70b62a7a89d681656c6413ce
Lung_Opacity-2986.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2986.png,Lung_Opacity/images/Lung_Opacity-2986.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2986.png,Lung_Opacity/masks/Lung_Opacity-2986.png,True,False,299,299,L,103.32,b40bfec02c37c5d390941be73bc4e66e
Lung_Opacity-2987.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2987.png,Lung_Opacity/images/Lung_Opacity-2987.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2987.png,Lung_Opacity/masks/Lung_Opacity-2987.png,True,False,299,299,L,133.3,c5e03c8d89ec1b38df16bd63edfe7c6f
Lung_Opacity-2988.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2988.png,Lung_Opacity/images/Lung_Opacity-2988.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2988.png,Lung_Opacity/masks/Lung_Opacity-2988.png,True,False,299,299,L,139.79,ee3729fd52aa3eefb3c55f7bd48f6942
Lung_Opacity-2989.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2989.png,Lung_Opacity/images/Lung_Opacity-2989.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2989.png,Lung_Opacity/masks/Lung_Opacity-2989.png,True,False,299,299,L,115.87,080523a517bf8c86fac5b764da95b7ce
Lung_Opacity-299.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-299.png,Lung_Opacity/images/Lung_Opacity-299.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-299.png,Lung_Opacity/masks/Lung_Opacity-299.png,True,False,299,299,L,130.24,65e73a61b8494e23c311ef5ab49093c5
Lung_Opacity-2990.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2990.png,Lung_Opacity/images/Lung_Opacity-2990.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2990.png,Lung_Opacity/masks/Lung_Opacity-2990.png,True,False,299,299,L,180.5,4e5ee88fda48f908fe45f3c2e45debf2
Lung_Opacity-2991.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2991.png,Lung_Opacity/images/Lung_Opacity-2991.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2991.png,Lung_Opacity/masks/Lung_Opacity-2991.png,True,False,299,299,L,112.63,74cedb1dec60e7690fbabcc73f5e04c7
Lung_Opacity-2992.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2992.png,Lung_Opacity/images/Lung_Opacity-2992.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2992.png,Lung_Opacity/masks/Lung_Opacity-2992.png,True,False,299,299,L,162.72,1573621ebb952e9e681ba4a6500ef67f
Lung_Opacity-2993.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2993.png,Lung_Opacity/images/Lung_Opacity-2993.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2993.png,Lung_Opacity/masks/Lung_Opacity-2993.png,True,False,299,299,L,107.24,3cfb0622209438bf56cd58c9e1faed6e
Lung_Opacity-2994.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2994.png,Lung_Opacity/images/Lung_Opacity-2994.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2994.png,Lung_Opacity/masks/Lung_Opacity-2994.png,True,False,299,299,L,61.5,053485f63311ed10c5f9184ab197e33f
Lung_Opacity-2995.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2995.png,Lung_Opacity/images/Lung_Opacity-2995.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2995.png,Lung_Opacity/masks/Lung_Opacity-2995.png,True,False,299,299,L,103.9,de101628850b21d8015a7669c164025c
Lung_Opacity-2996.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2996.png,Lung_Opacity/images/Lung_Opacity-2996.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2996.png,Lung_Opacity/masks/Lung_Opacity-2996.png,True,False,299,299,L,161.59,9c8a00517323bc22c6f340f71ec62f75
Lung_Opacity-2997.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2997.png,Lung_Opacity/images/Lung_Opacity-2997.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2997.png,Lung_Opacity/masks/Lung_Opacity-2997.png,True,False,299,299,L,116.84,14c11eed2a65177f5f8338fd235024ed
Lung_Opacity-2998.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2998.png,Lung_Opacity/images/Lung_Opacity-2998.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2998.png,Lung_Opacity/masks/Lung_Opacity-2998.png,True,False,299,299,L,114.68,09fd018350f103333a4ced8a46f2c5c4
Lung_Opacity-2999.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-2999.png,Lung_Opacity/images/Lung_Opacity-2999.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-2999.png,Lung_Opacity/masks/Lung_Opacity-2999.png,True,False,299,299,L,116.57,8c9eefb35bd18d607e313d9ec60089b6
Lung_Opacity-3.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3.png,Lung_Opacity/images/Lung_Opacity-3.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3.png,Lung_Opacity/masks/Lung_Opacity-3.png,True,False,299,299,L,120.25,1f16201a039f37453e0968eb8af07125
Lung_Opacity-30.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-30.png,Lung_Opacity/images/Lung_Opacity-30.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-30.png,Lung_Opacity/masks/Lung_Opacity-30.png,True,False,299,299,L,146.19,01fb532fe31d3c15ca3e04995a5d51bd
Lung_Opacity-300.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-300.png,Lung_Opacity/images/Lung_Opacity-300.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-300.png,Lung_Opacity/masks/Lung_Opacity-300.png,True,False,299,299,L,137.43,fc5f6f333593e7cc66deaa75c8f53c94
Lung_Opacity-3000.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3000.png,Lung_Opacity/images/Lung_Opacity-3000.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3000.png,Lung_Opacity/masks/Lung_Opacity-3000.png,True,False,299,299,L,114.82,efd1ea7118f0af22a6053cf28984b95c
Lung_Opacity-3001.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3001.png,Lung_Opacity/images/Lung_Opacity-3001.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3001.png,Lung_Opacity/masks/Lung_Opacity-3001.png,True,False,299,299,L,151.56,449b0a2fd278dceb9532137376e9b756
Lung_Opacity-3002.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3002.png,Lung_Opacity/images/Lung_Opacity-3002.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3002.png,Lung_Opacity/masks/Lung_Opacity-3002.png,True,False,299,299,L,149.18,e2d213912cc86d5d70c594eebc9bc71a
Lung_Opacity-3003.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3003.png,Lung_Opacity/images/Lung_Opacity-3003.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3003.png,Lung_Opacity/masks/Lung_Opacity-3003.png,True,False,299,299,L,110.71,07977c8d2999ddb8dbedf86019561447
Lung_Opacity-3004.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3004.png,Lung_Opacity/images/Lung_Opacity-3004.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3004.png,Lung_Opacity/masks/Lung_Opacity-3004.png,True,False,299,299,L,113.49,064d3de2174aabbb70a0df771dc86305
Lung_Opacity-3005.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3005.png,Lung_Opacity/images/Lung_Opacity-3005.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3005.png,Lung_Opacity/masks/Lung_Opacity-3005.png,True,False,299,299,L,118.17,9e63cee20093b4e367427b81e09c3390
Lung_Opacity-3006.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3006.png,Lung_Opacity/images/Lung_Opacity-3006.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3006.png,Lung_Opacity/masks/Lung_Opacity-3006.png,True,False,299,299,L,156.27,c7d6bfe8d344504d2dfc69e317525fae
Lung_Opacity-3007.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3007.png,Lung_Opacity/images/Lung_Opacity-3007.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3007.png,Lung_Opacity/masks/Lung_Opacity-3007.png,True,False,299,299,L,109.46,b2f5b0673267745d606a1de047c33380
Lung_Opacity-3008.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3008.png,Lung_Opacity/images/Lung_Opacity-3008.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3008.png,Lung_Opacity/masks/Lung_Opacity-3008.png,True,False,299,299,L,115.39,0a85f0878f77fead9815f904b7611cc1
Lung_Opacity-3009.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3009.png,Lung_Opacity/images/Lung_Opacity-3009.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3009.png,Lung_Opacity/masks/Lung_Opacity-3009.png,True,False,299,299,L,123.86,4b6b7abf0be79992523ad4af5b2ab0a7
Lung_Opacity-301.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-301.png,Lung_Opacity/images/Lung_Opacity-301.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-301.png,Lung_Opacity/masks/Lung_Opacity-301.png,True,False,299,299,L,160.5,a61ba72d8754be6d4f1a40cf12721ac6
Lung_Opacity-3010.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3010.png,Lung_Opacity/images/Lung_Opacity-3010.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3010.png,Lung_Opacity/masks/Lung_Opacity-3010.png,True,False,299,299,L,118.64,8b43c6be0331b932236ca2a332bc8090
Lung_Opacity-3011.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3011.png,Lung_Opacity/images/Lung_Opacity-3011.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3011.png,Lung_Opacity/masks/Lung_Opacity-3011.png,True,False,299,299,L,111.6,78745187871185ed672444b504268953
Lung_Opacity-3012.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3012.png,Lung_Opacity/images/Lung_Opacity-3012.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3012.png,Lung_Opacity/masks/Lung_Opacity-3012.png,True,False,299,299,L,147.66,48b64eeb39f7c1fa7574b9ade6483d3a
Lung_Opacity-3013.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3013.png,Lung_Opacity/images/Lung_Opacity-3013.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3013.png,Lung_Opacity/masks/Lung_Opacity-3013.png,True,False,299,299,L,152.37,1c9b893365bfa04f3dfe241e1d97dda3
Lung_Opacity-3014.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3014.png,Lung_Opacity/images/Lung_Opacity-3014.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3014.png,Lung_Opacity/masks/Lung_Opacity-3014.png,True,False,299,299,L,158.39,99ebe14a01dddb03642b2d56a69dbf56
Lung_Opacity-3015.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3015.png,Lung_Opacity/images/Lung_Opacity-3015.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3015.png,Lung_Opacity/masks/Lung_Opacity-3015.png,True,False,299,299,L,131.43,0b9be4839169ce3d9bbd1bf998a3f9f0
Lung_Opacity-3016.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3016.png,Lung_Opacity/images/Lung_Opacity-3016.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3016.png,Lung_Opacity/masks/Lung_Opacity-3016.png,True,False,299,299,L,115.66,a032252023d21a960abc623ffc610886
Lung_Opacity-3017.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3017.png,Lung_Opacity/images/Lung_Opacity-3017.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3017.png,Lung_Opacity/masks/Lung_Opacity-3017.png,True,False,299,299,L,122.92,b30a8a9fefa5b7ec02daebf1a63cd565
Lung_Opacity-3018.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3018.png,Lung_Opacity/images/Lung_Opacity-3018.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3018.png,Lung_Opacity/masks/Lung_Opacity-3018.png,True,False,299,299,L,157.76,67844c7ab18fddff22bf64da2d467ed0
Lung_Opacity-3019.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3019.png,Lung_Opacity/images/Lung_Opacity-3019.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3019.png,Lung_Opacity/masks/Lung_Opacity-3019.png,True,False,299,299,L,110.48,ea1a601cdc8ffd669bdfac724239ebe2
Lung_Opacity-302.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-302.png,Lung_Opacity/images/Lung_Opacity-302.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-302.png,Lung_Opacity/masks/Lung_Opacity-302.png,True,False,299,299,L,114.78,d0dd969f1433cba6ee9b4229085fa573
Lung_Opacity-3020.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3020.png,Lung_Opacity/images/Lung_Opacity-3020.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3020.png,Lung_Opacity/masks/Lung_Opacity-3020.png,True,False,299,299,L,128.76,606c4fc5a5f1edc3b5d4c4d962f99d3c
Lung_Opacity-3021.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3021.png,Lung_Opacity/images/Lung_Opacity-3021.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3021.png,Lung_Opacity/masks/Lung_Opacity-3021.png,True,False,299,299,L,122.46,4c441ff3e80b464976dee6982493053a
Lung_Opacity-3022.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3022.png,Lung_Opacity/images/Lung_Opacity-3022.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3022.png,Lung_Opacity/masks/Lung_Opacity-3022.png,True,False,299,299,L,124.02,f3ae874e3c39c52375b465cd1cb681a2
Lung_Opacity-3023.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3023.png,Lung_Opacity/images/Lung_Opacity-3023.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3023.png,Lung_Opacity/masks/Lung_Opacity-3023.png,True,False,299,299,L,145.81,7136a07a451abf19808a31d5354b8b64
Lung_Opacity-3024.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3024.png,Lung_Opacity/images/Lung_Opacity-3024.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3024.png,Lung_Opacity/masks/Lung_Opacity-3024.png,True,False,299,299,L,100.2,b44c107e475d21fb2391d593c759fc83
Lung_Opacity-3025.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3025.png,Lung_Opacity/images/Lung_Opacity-3025.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3025.png,Lung_Opacity/masks/Lung_Opacity-3025.png,True,False,299,299,L,113.71,c6cd2a2ffeaaeebde77a68485c4cfae2
Lung_Opacity-3026.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3026.png,Lung_Opacity/images/Lung_Opacity-3026.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3026.png,Lung_Opacity/masks/Lung_Opacity-3026.png,True,False,299,299,L,114.44,30f97f56394bbcab3ea871d1a82a7a30
Lung_Opacity-3027.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3027.png,Lung_Opacity/images/Lung_Opacity-3027.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3027.png,Lung_Opacity/masks/Lung_Opacity-3027.png,True,False,299,299,L,143.52,a27c06550921b00d2937fdb21dded195
Lung_Opacity-3028.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3028.png,Lung_Opacity/images/Lung_Opacity-3028.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3028.png,Lung_Opacity/masks/Lung_Opacity-3028.png,True,False,299,299,L,152.49,4bcb90c4b9decfacd17dbb19cf08eb28
Lung_Opacity-3029.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3029.png,Lung_Opacity/images/Lung_Opacity-3029.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3029.png,Lung_Opacity/masks/Lung_Opacity-3029.png,True,False,299,299,L,109.24,06e53e86518926096d84e78a658fbe1f
Lung_Opacity-303.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-303.png,Lung_Opacity/images/Lung_Opacity-303.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-303.png,Lung_Opacity/masks/Lung_Opacity-303.png,True,False,299,299,L,158.11,ee9f67b4663e05c2e5912d33305d4d22
Lung_Opacity-3030.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3030.png,Lung_Opacity/images/Lung_Opacity-3030.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3030.png,Lung_Opacity/masks/Lung_Opacity-3030.png,True,False,299,299,L,103.34,c18186414ec262a8eab32d252f14f2d3
Lung_Opacity-3031.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3031.png,Lung_Opacity/images/Lung_Opacity-3031.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3031.png,Lung_Opacity/masks/Lung_Opacity-3031.png,True,False,299,299,L,146.31,f2262d7e1546a44a2fe3cd9b8d0ef600
Lung_Opacity-3032.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3032.png,Lung_Opacity/images/Lung_Opacity-3032.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3032.png,Lung_Opacity/masks/Lung_Opacity-3032.png,True,False,299,299,L,107.51,a77399b9d3b945d16075276ad79ac44c
Lung_Opacity-3033.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3033.png,Lung_Opacity/images/Lung_Opacity-3033.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3033.png,Lung_Opacity/masks/Lung_Opacity-3033.png,True,False,299,299,L,157.76,35623e11b02327b664025bdc389753d8
Lung_Opacity-3034.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3034.png,Lung_Opacity/images/Lung_Opacity-3034.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3034.png,Lung_Opacity/masks/Lung_Opacity-3034.png,True,False,299,299,L,108.82,5b059b5f267eb6f839353b92fe0a9d2b
Lung_Opacity-3035.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3035.png,Lung_Opacity/images/Lung_Opacity-3035.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3035.png,Lung_Opacity/masks/Lung_Opacity-3035.png,True,False,299,299,L,109.42,7a097d58ac8f4c11d9594c1ece9e2a43
Lung_Opacity-3036.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3036.png,Lung_Opacity/images/Lung_Opacity-3036.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3036.png,Lung_Opacity/masks/Lung_Opacity-3036.png,True,False,299,299,L,159.44,86e92fcd515dfda843efe4f615b7d11b
Lung_Opacity-3037.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3037.png,Lung_Opacity/images/Lung_Opacity-3037.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3037.png,Lung_Opacity/masks/Lung_Opacity-3037.png,True,False,299,299,L,86.86,72b38451fa913d71e80952bc9bef85bf
Lung_Opacity-3038.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3038.png,Lung_Opacity/images/Lung_Opacity-3038.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3038.png,Lung_Opacity/masks/Lung_Opacity-3038.png,True,False,299,299,L,142.87,1fa805517d755ce5fa2f8662af176beb
Lung_Opacity-3039.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3039.png,Lung_Opacity/images/Lung_Opacity-3039.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3039.png,Lung_Opacity/masks/Lung_Opacity-3039.png,True,False,299,299,L,104.21,eaba127ff07fe0d8252411fbf2f1e85c
Lung_Opacity-304.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-304.png,Lung_Opacity/images/Lung_Opacity-304.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-304.png,Lung_Opacity/masks/Lung_Opacity-304.png,True,False,299,299,L,171.5,c36ca72b0118bd19fbbaabdfa9f799c4
Lung_Opacity-3040.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3040.png,Lung_Opacity/images/Lung_Opacity-3040.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3040.png,Lung_Opacity/masks/Lung_Opacity-3040.png,True,False,299,299,L,119.44,c4a223bd56de1896e93cc1a88742436e
Lung_Opacity-3041.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3041.png,Lung_Opacity/images/Lung_Opacity-3041.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3041.png,Lung_Opacity/masks/Lung_Opacity-3041.png,True,False,299,299,L,124.13,9dfe999f5b81789a26867d5aaf741075
Lung_Opacity-3042.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3042.png,Lung_Opacity/images/Lung_Opacity-3042.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3042.png,Lung_Opacity/masks/Lung_Opacity-3042.png,True,False,299,299,L,109.59,3a7a991b5bc6d2dfc6a31642601cb151
Lung_Opacity-3043.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3043.png,Lung_Opacity/images/Lung_Opacity-3043.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3043.png,Lung_Opacity/masks/Lung_Opacity-3043.png,True,False,299,299,L,100.08,114ee7f3abd1caa5ace4d5adf41522a9
Lung_Opacity-3044.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3044.png,Lung_Opacity/images/Lung_Opacity-3044.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3044.png,Lung_Opacity/masks/Lung_Opacity-3044.png,True,False,299,299,L,97.17,c792bc20d409e143f9744d34a2d95e96
Lung_Opacity-3045.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3045.png,Lung_Opacity/images/Lung_Opacity-3045.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3045.png,Lung_Opacity/masks/Lung_Opacity-3045.png,True,False,299,299,L,110.17,27a9d7fb1d2db6780f56463767cad244
Lung_Opacity-3046.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3046.png,Lung_Opacity/images/Lung_Opacity-3046.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3046.png,Lung_Opacity/masks/Lung_Opacity-3046.png,True,False,299,299,L,150.42,df0a684739eb8c1a3df2217f5fcd10b3
Lung_Opacity-3047.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3047.png,Lung_Opacity/images/Lung_Opacity-3047.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3047.png,Lung_Opacity/masks/Lung_Opacity-3047.png,True,False,299,299,L,108.91,276b1c7169c2112904a6eb59c705e219
Lung_Opacity-3048.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3048.png,Lung_Opacity/images/Lung_Opacity-3048.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3048.png,Lung_Opacity/masks/Lung_Opacity-3048.png,True,False,299,299,L,127.02,15cca3007d1ad7154478ae4fcd20baeb
Lung_Opacity-3049.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3049.png,Lung_Opacity/images/Lung_Opacity-3049.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3049.png,Lung_Opacity/masks/Lung_Opacity-3049.png,True,False,299,299,L,148.38,2413b5f44eda92a3c865b22e481f3f65
Lung_Opacity-305.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-305.png,Lung_Opacity/images/Lung_Opacity-305.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-305.png,Lung_Opacity/masks/Lung_Opacity-305.png,True,False,299,299,L,164.75,cd9b4337d34b8ebb91aaa7c884c1af02
Lung_Opacity-3050.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3050.png,Lung_Opacity/images/Lung_Opacity-3050.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3050.png,Lung_Opacity/masks/Lung_Opacity-3050.png,True,False,299,299,L,103.52,162df812cf993bd01369e5f23d1ddaf3
Lung_Opacity-3051.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3051.png,Lung_Opacity/images/Lung_Opacity-3051.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3051.png,Lung_Opacity/masks/Lung_Opacity-3051.png,True,False,299,299,L,120.79,6c183fbefcdbc7266fe6177979c552dd
Lung_Opacity-3052.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3052.png,Lung_Opacity/images/Lung_Opacity-3052.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3052.png,Lung_Opacity/masks/Lung_Opacity-3052.png,True,False,299,299,L,182.24,d23a4e99769a28b47e723cd9666537f3
Lung_Opacity-3053.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3053.png,Lung_Opacity/images/Lung_Opacity-3053.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3053.png,Lung_Opacity/masks/Lung_Opacity-3053.png,True,False,299,299,L,103.53,d7c789e25369ef566fb8b97f655c8f6c
Lung_Opacity-3054.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3054.png,Lung_Opacity/images/Lung_Opacity-3054.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3054.png,Lung_Opacity/masks/Lung_Opacity-3054.png,True,False,299,299,L,149.03,9842a8fbedf474971ce8e4b66c93be91
Lung_Opacity-3055.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3055.png,Lung_Opacity/images/Lung_Opacity-3055.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3055.png,Lung_Opacity/masks/Lung_Opacity-3055.png,True,False,299,299,L,115.08,2650fbde2fae1fe5dbe77a6ef9193def
Lung_Opacity-3056.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3056.png,Lung_Opacity/images/Lung_Opacity-3056.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3056.png,Lung_Opacity/masks/Lung_Opacity-3056.png,True,False,299,299,L,107.8,fc76cca6a982d565b2d999c896004eac
Lung_Opacity-3057.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3057.png,Lung_Opacity/images/Lung_Opacity-3057.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3057.png,Lung_Opacity/masks/Lung_Opacity-3057.png,True,False,299,299,L,93.7,c0e222c1b94ccb70e30e74f599c9ea72
Lung_Opacity-3058.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3058.png,Lung_Opacity/images/Lung_Opacity-3058.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3058.png,Lung_Opacity/masks/Lung_Opacity-3058.png,True,False,299,299,L,118.56,31cf2f093d585f747b23e47663ed168e
Lung_Opacity-3059.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3059.png,Lung_Opacity/images/Lung_Opacity-3059.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3059.png,Lung_Opacity/masks/Lung_Opacity-3059.png,True,False,299,299,L,112.33,8dd584ccb7a49649e301db049cd1b3ad
Lung_Opacity-306.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-306.png,Lung_Opacity/images/Lung_Opacity-306.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-306.png,Lung_Opacity/masks/Lung_Opacity-306.png,True,False,299,299,L,131.96,6546ace99da99c7cec7883e143693c01
Lung_Opacity-3060.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3060.png,Lung_Opacity/images/Lung_Opacity-3060.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3060.png,Lung_Opacity/masks/Lung_Opacity-3060.png,True,False,299,299,L,88.13,5fec2b5b205a79ac35cd5725f89d4f86
Lung_Opacity-3061.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3061.png,Lung_Opacity/images/Lung_Opacity-3061.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3061.png,Lung_Opacity/masks/Lung_Opacity-3061.png,True,False,299,299,L,135.96,940ba396ccb4ad10bf4244594d898301
Lung_Opacity-3062.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3062.png,Lung_Opacity/images/Lung_Opacity-3062.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3062.png,Lung_Opacity/masks/Lung_Opacity-3062.png,True,False,299,299,L,189.08,6441ec85246e7e6b068f8e77c47fe9de
Lung_Opacity-3063.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3063.png,Lung_Opacity/images/Lung_Opacity-3063.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3063.png,Lung_Opacity/masks/Lung_Opacity-3063.png,True,False,299,299,L,124.73,23d3d9e41f2a324030ce7b0c0add368c
Lung_Opacity-3064.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3064.png,Lung_Opacity/images/Lung_Opacity-3064.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3064.png,Lung_Opacity/masks/Lung_Opacity-3064.png,True,False,299,299,L,136.07,7bfe3dc8a39bab377e9a328a917319ff
Lung_Opacity-3065.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3065.png,Lung_Opacity/images/Lung_Opacity-3065.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3065.png,Lung_Opacity/masks/Lung_Opacity-3065.png,True,False,299,299,L,135.95,3a50ef530d22d51c9a7671f83af8daf8
Lung_Opacity-3066.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3066.png,Lung_Opacity/images/Lung_Opacity-3066.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3066.png,Lung_Opacity/masks/Lung_Opacity-3066.png,True,False,299,299,L,164.84,6f21a8f54200c2234f7924baf8127017
Lung_Opacity-3067.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3067.png,Lung_Opacity/images/Lung_Opacity-3067.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3067.png,Lung_Opacity/masks/Lung_Opacity-3067.png,True,False,299,299,L,143.08,fc88aaf5bba79075964eea7e49e123c0
Lung_Opacity-3068.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3068.png,Lung_Opacity/images/Lung_Opacity-3068.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3068.png,Lung_Opacity/masks/Lung_Opacity-3068.png,True,False,299,299,L,106.48,0fef04fb90e3da6c998b1a4faed4839b
Lung_Opacity-3069.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3069.png,Lung_Opacity/images/Lung_Opacity-3069.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3069.png,Lung_Opacity/masks/Lung_Opacity-3069.png,True,False,299,299,L,107.0,38890d60c4f6b85fdb75b542fd01d3c4
Lung_Opacity-307.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-307.png,Lung_Opacity/images/Lung_Opacity-307.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-307.png,Lung_Opacity/masks/Lung_Opacity-307.png,True,False,299,299,L,139.26,7b641e19ed766003c08f03ffa3132991
Lung_Opacity-3070.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3070.png,Lung_Opacity/images/Lung_Opacity-3070.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3070.png,Lung_Opacity/masks/Lung_Opacity-3070.png,True,False,299,299,L,124.26,74bef451e15f51d30451c4d35667db55
Lung_Opacity-3071.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3071.png,Lung_Opacity/images/Lung_Opacity-3071.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3071.png,Lung_Opacity/masks/Lung_Opacity-3071.png,True,False,299,299,L,111.97,5298a38b71a1fea90a1d63ad4ff2a753
Lung_Opacity-3072.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3072.png,Lung_Opacity/images/Lung_Opacity-3072.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3072.png,Lung_Opacity/masks/Lung_Opacity-3072.png,True,False,299,299,L,172.89,2033e343738bd455285da7476a1c8070
Lung_Opacity-3073.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3073.png,Lung_Opacity/images/Lung_Opacity-3073.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3073.png,Lung_Opacity/masks/Lung_Opacity-3073.png,True,False,299,299,L,110.89,f460269eb1992e79535a92c492bf5d80
Lung_Opacity-3074.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3074.png,Lung_Opacity/images/Lung_Opacity-3074.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3074.png,Lung_Opacity/masks/Lung_Opacity-3074.png,True,False,299,299,L,126.04,bd75a6508e752e850cae04dd4ad1eaf5
Lung_Opacity-3075.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3075.png,Lung_Opacity/images/Lung_Opacity-3075.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3075.png,Lung_Opacity/masks/Lung_Opacity-3075.png,True,False,299,299,L,172.75,249b4feaba98b803bd9231a497a06b19
Lung_Opacity-3076.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3076.png,Lung_Opacity/images/Lung_Opacity-3076.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3076.png,Lung_Opacity/masks/Lung_Opacity-3076.png,True,False,299,299,L,109.0,5517597bb267b82dc22586787917e129
Lung_Opacity-3077.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3077.png,Lung_Opacity/images/Lung_Opacity-3077.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3077.png,Lung_Opacity/masks/Lung_Opacity-3077.png,True,False,299,299,L,116.8,521230d2f673f658ea3a191199c33076
Lung_Opacity-3078.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3078.png,Lung_Opacity/images/Lung_Opacity-3078.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3078.png,Lung_Opacity/masks/Lung_Opacity-3078.png,True,False,299,299,L,141.46,ef05b362581a0f7ed4dc0f2b6a77e974
Lung_Opacity-3079.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3079.png,Lung_Opacity/images/Lung_Opacity-3079.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3079.png,Lung_Opacity/masks/Lung_Opacity-3079.png,True,False,299,299,L,137.73,5a5f3596bcc4c92102f520a3e77d4400
Lung_Opacity-308.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-308.png,Lung_Opacity/images/Lung_Opacity-308.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-308.png,Lung_Opacity/masks/Lung_Opacity-308.png,True,False,299,299,L,117.19,de260e60237025b5b975a761410f091c
Lung_Opacity-3080.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3080.png,Lung_Opacity/images/Lung_Opacity-3080.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3080.png,Lung_Opacity/masks/Lung_Opacity-3080.png,True,False,299,299,L,106.44,d968420b1136f9ddddf6b51144a2742a
Lung_Opacity-3081.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3081.png,Lung_Opacity/images/Lung_Opacity-3081.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3081.png,Lung_Opacity/masks/Lung_Opacity-3081.png,True,False,299,299,L,112.57,57c44e5936b92aae0c771320702e9afa
Lung_Opacity-3082.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3082.png,Lung_Opacity/images/Lung_Opacity-3082.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3082.png,Lung_Opacity/masks/Lung_Opacity-3082.png,True,False,299,299,L,125.78,3cf9094bc50d12f18355e2d775e0059b
Lung_Opacity-3083.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3083.png,Lung_Opacity/images/Lung_Opacity-3083.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3083.png,Lung_Opacity/masks/Lung_Opacity-3083.png,True,False,299,299,L,116.56,04b593f4087edc1145851dd7eccbd76f
Lung_Opacity-3084.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3084.png,Lung_Opacity/images/Lung_Opacity-3084.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3084.png,Lung_Opacity/masks/Lung_Opacity-3084.png,True,False,299,299,L,122.82,369b20d8acf148b2e566e76fff2962a7
Lung_Opacity-3085.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3085.png,Lung_Opacity/images/Lung_Opacity-3085.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3085.png,Lung_Opacity/masks/Lung_Opacity-3085.png,True,False,299,299,L,181.97,8bcd726845461c80147443cb6095ae7a
Lung_Opacity-3086.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3086.png,Lung_Opacity/images/Lung_Opacity-3086.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3086.png,Lung_Opacity/masks/Lung_Opacity-3086.png,True,False,299,299,L,117.58,05cd142433ba2a7ae4128ec401af77e1
Lung_Opacity-3087.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3087.png,Lung_Opacity/images/Lung_Opacity-3087.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3087.png,Lung_Opacity/masks/Lung_Opacity-3087.png,True,False,299,299,L,145.17,0317d6a68c318afae9206533dcc37c51
Lung_Opacity-3088.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3088.png,Lung_Opacity/images/Lung_Opacity-3088.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3088.png,Lung_Opacity/masks/Lung_Opacity-3088.png,True,False,299,299,L,129.07,8dda78de288e26275e5a2e552063ea62
Lung_Opacity-3089.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3089.png,Lung_Opacity/images/Lung_Opacity-3089.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3089.png,Lung_Opacity/masks/Lung_Opacity-3089.png,True,False,299,299,L,109.3,31195d60db762aff31091e89676b037e
Lung_Opacity-309.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-309.png,Lung_Opacity/images/Lung_Opacity-309.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-309.png,Lung_Opacity/masks/Lung_Opacity-309.png,True,False,299,299,L,109.25,d411088360406392e6179b4a13839074
Lung_Opacity-3090.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3090.png,Lung_Opacity/images/Lung_Opacity-3090.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3090.png,Lung_Opacity/masks/Lung_Opacity-3090.png,True,False,299,299,L,116.47,088171ea09974e80e888acbca826d9ec
Lung_Opacity-3091.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3091.png,Lung_Opacity/images/Lung_Opacity-3091.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3091.png,Lung_Opacity/masks/Lung_Opacity-3091.png,True,False,299,299,L,151.14,2bdccfb64d3c51d5c00aab1a83a719f1
Lung_Opacity-3092.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3092.png,Lung_Opacity/images/Lung_Opacity-3092.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3092.png,Lung_Opacity/masks/Lung_Opacity-3092.png,True,False,299,299,L,110.92,3cdb02015f1b84fa8d66f04a9bf63b09
Lung_Opacity-3093.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3093.png,Lung_Opacity/images/Lung_Opacity-3093.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3093.png,Lung_Opacity/masks/Lung_Opacity-3093.png,True,False,299,299,L,127.43,aa8566ca999eb843f6a4c560750e915b
Lung_Opacity-3094.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3094.png,Lung_Opacity/images/Lung_Opacity-3094.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3094.png,Lung_Opacity/masks/Lung_Opacity-3094.png,True,False,299,299,L,132.46,b1f82b5840b78f55bf6ef5982f007e69
Lung_Opacity-3095.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3095.png,Lung_Opacity/images/Lung_Opacity-3095.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3095.png,Lung_Opacity/masks/Lung_Opacity-3095.png,True,False,299,299,L,124.77,e39fc12a3dcda13c4ded8851fe52f175
Lung_Opacity-3096.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3096.png,Lung_Opacity/images/Lung_Opacity-3096.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3096.png,Lung_Opacity/masks/Lung_Opacity-3096.png,True,False,299,299,L,128.18,0b4ce7d52be7359d2fe5c5c08d743182
Lung_Opacity-3097.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3097.png,Lung_Opacity/images/Lung_Opacity-3097.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3097.png,Lung_Opacity/masks/Lung_Opacity-3097.png,True,False,299,299,L,124.56,42fd04f11106255415aa8156b5a15f62
Lung_Opacity-3098.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3098.png,Lung_Opacity/images/Lung_Opacity-3098.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3098.png,Lung_Opacity/masks/Lung_Opacity-3098.png,True,False,299,299,L,140.01,97c79eac97a6eca672483cb2c6a63491
Lung_Opacity-3099.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3099.png,Lung_Opacity/images/Lung_Opacity-3099.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3099.png,Lung_Opacity/masks/Lung_Opacity-3099.png,True,False,299,299,L,129.19,ce42492235d7e91d09abd3de06fcf937
Lung_Opacity-31.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-31.png,Lung_Opacity/images/Lung_Opacity-31.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-31.png,Lung_Opacity/masks/Lung_Opacity-31.png,True,False,299,299,L,110.16,3245a53ee3e95c7386ef5a982dc2bf69
Lung_Opacity-310.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-310.png,Lung_Opacity/images/Lung_Opacity-310.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-310.png,Lung_Opacity/masks/Lung_Opacity-310.png,True,False,299,299,L,111.76,0366d48c6a1711fd560efe27849cd7d0
Lung_Opacity-3100.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3100.png,Lung_Opacity/images/Lung_Opacity-3100.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3100.png,Lung_Opacity/masks/Lung_Opacity-3100.png,True,False,299,299,L,143.11,af76138d56aee5d254e12383ccfcb4ef
Lung_Opacity-3101.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3101.png,Lung_Opacity/images/Lung_Opacity-3101.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3101.png,Lung_Opacity/masks/Lung_Opacity-3101.png,True,False,299,299,L,116.44,beb6d3a3dcb6fca6880332238b0f7d72
Lung_Opacity-3102.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3102.png,Lung_Opacity/images/Lung_Opacity-3102.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3102.png,Lung_Opacity/masks/Lung_Opacity-3102.png,True,False,299,299,L,95.82,da7a907e545d10ab19bb2540429903da
Lung_Opacity-3103.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3103.png,Lung_Opacity/images/Lung_Opacity-3103.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3103.png,Lung_Opacity/masks/Lung_Opacity-3103.png,True,False,299,299,L,141.71,7450e24a9e7193506d9792efb6545ba3
Lung_Opacity-3104.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3104.png,Lung_Opacity/images/Lung_Opacity-3104.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3104.png,Lung_Opacity/masks/Lung_Opacity-3104.png,True,False,299,299,L,127.38,e53a3701a4b9d82a9e8ef315cd901abf
Lung_Opacity-3105.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3105.png,Lung_Opacity/images/Lung_Opacity-3105.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3105.png,Lung_Opacity/masks/Lung_Opacity-3105.png,True,False,299,299,L,121.91,0f8c1194756111a8536b101fa996b792
Lung_Opacity-3106.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3106.png,Lung_Opacity/images/Lung_Opacity-3106.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3106.png,Lung_Opacity/masks/Lung_Opacity-3106.png,True,False,299,299,L,158.24,56c6c3c61dd6bd877a4131d34c7f7071
Lung_Opacity-3107.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3107.png,Lung_Opacity/images/Lung_Opacity-3107.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3107.png,Lung_Opacity/masks/Lung_Opacity-3107.png,True,False,299,299,L,100.65,4f3ef21851a9ef2661685762c3696141
Lung_Opacity-3108.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3108.png,Lung_Opacity/images/Lung_Opacity-3108.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3108.png,Lung_Opacity/masks/Lung_Opacity-3108.png,True,False,299,299,L,132.44,73abd1fe1af7b7fcb56006e5e2bfbb67
Lung_Opacity-3109.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3109.png,Lung_Opacity/images/Lung_Opacity-3109.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3109.png,Lung_Opacity/masks/Lung_Opacity-3109.png,True,False,299,299,L,108.76,e78e105f2c80f511468c6abad1b1ba35
Lung_Opacity-311.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-311.png,Lung_Opacity/images/Lung_Opacity-311.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-311.png,Lung_Opacity/masks/Lung_Opacity-311.png,True,False,299,299,L,87.9,bc79c858cb465754b55113cb1e3195fb
Lung_Opacity-3110.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3110.png,Lung_Opacity/images/Lung_Opacity-3110.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3110.png,Lung_Opacity/masks/Lung_Opacity-3110.png,True,False,299,299,L,115.4,b81b4b7387e6a57a1d86eb79fa27c349
Lung_Opacity-3111.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3111.png,Lung_Opacity/images/Lung_Opacity-3111.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3111.png,Lung_Opacity/masks/Lung_Opacity-3111.png,True,False,299,299,L,113.45,5378b4db2a9202d222c08171e1faf9ad
Lung_Opacity-3112.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3112.png,Lung_Opacity/images/Lung_Opacity-3112.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3112.png,Lung_Opacity/masks/Lung_Opacity-3112.png,True,False,299,299,L,171.72,d242460e37e629feff6055820ed86469
Lung_Opacity-3113.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3113.png,Lung_Opacity/images/Lung_Opacity-3113.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3113.png,Lung_Opacity/masks/Lung_Opacity-3113.png,True,False,299,299,L,105.41,e6d535b47bfc61920b47685b405892d9
Lung_Opacity-3114.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3114.png,Lung_Opacity/images/Lung_Opacity-3114.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3114.png,Lung_Opacity/masks/Lung_Opacity-3114.png,True,False,299,299,L,172.58,936ac14e716a27817270af37cd8b2263
Lung_Opacity-3115.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3115.png,Lung_Opacity/images/Lung_Opacity-3115.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3115.png,Lung_Opacity/masks/Lung_Opacity-3115.png,True,False,299,299,L,114.45,98170fd293916167aeed1dc4a507ab98
Lung_Opacity-3116.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3116.png,Lung_Opacity/images/Lung_Opacity-3116.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3116.png,Lung_Opacity/masks/Lung_Opacity-3116.png,True,False,299,299,L,131.29,a2766f0875850f673724486340cb3f0e
Lung_Opacity-3117.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3117.png,Lung_Opacity/images/Lung_Opacity-3117.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3117.png,Lung_Opacity/masks/Lung_Opacity-3117.png,True,False,299,299,L,141.17,74995b2f22af071c49cb15eeccbca870
Lung_Opacity-3118.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3118.png,Lung_Opacity/images/Lung_Opacity-3118.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3118.png,Lung_Opacity/masks/Lung_Opacity-3118.png,True,False,299,299,L,131.97,33accb70bea3e0870611687cf6a9ccf4
Lung_Opacity-3119.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3119.png,Lung_Opacity/images/Lung_Opacity-3119.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3119.png,Lung_Opacity/masks/Lung_Opacity-3119.png,True,False,299,299,L,161.37,c0b65a800d3ea7334dab37a4fa040ba5
Lung_Opacity-312.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-312.png,Lung_Opacity/images/Lung_Opacity-312.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-312.png,Lung_Opacity/masks/Lung_Opacity-312.png,True,False,299,299,L,167.48,0a1341f0fd240e2e26372c7bbe196285
Lung_Opacity-3120.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3120.png,Lung_Opacity/images/Lung_Opacity-3120.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3120.png,Lung_Opacity/masks/Lung_Opacity-3120.png,True,False,299,299,L,148.53,f1f75d069c725969eb7746dd8a5c7b90
Lung_Opacity-3121.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3121.png,Lung_Opacity/images/Lung_Opacity-3121.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3121.png,Lung_Opacity/masks/Lung_Opacity-3121.png,True,False,299,299,L,160.42,52922d01e6b6d7761ebd54dfefe738a0
Lung_Opacity-3122.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3122.png,Lung_Opacity/images/Lung_Opacity-3122.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3122.png,Lung_Opacity/masks/Lung_Opacity-3122.png,True,False,299,299,L,112.15,0eb807a2b6add35036d2131ff2564323
Lung_Opacity-3123.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3123.png,Lung_Opacity/images/Lung_Opacity-3123.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3123.png,Lung_Opacity/masks/Lung_Opacity-3123.png,True,False,299,299,L,129.21,67b4e1fa31e1c7c38676e0f84d275266
Lung_Opacity-3124.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3124.png,Lung_Opacity/images/Lung_Opacity-3124.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3124.png,Lung_Opacity/masks/Lung_Opacity-3124.png,True,False,299,299,L,106.28,c292fd39fcafefc70ba71ca5d15aca7d
Lung_Opacity-3125.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3125.png,Lung_Opacity/images/Lung_Opacity-3125.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3125.png,Lung_Opacity/masks/Lung_Opacity-3125.png,True,False,299,299,L,127.23,6d8ee6f5a822bef31065b9d5dc94c881
Lung_Opacity-3126.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3126.png,Lung_Opacity/images/Lung_Opacity-3126.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3126.png,Lung_Opacity/masks/Lung_Opacity-3126.png,True,False,299,299,L,87.04,5a494e051e5c43e38a5e67bb4c93fe99
Lung_Opacity-3127.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3127.png,Lung_Opacity/images/Lung_Opacity-3127.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3127.png,Lung_Opacity/masks/Lung_Opacity-3127.png,True,False,299,299,L,101.46,9cdaef2276b64f94dbd8eee43afbd731
Lung_Opacity-3128.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3128.png,Lung_Opacity/images/Lung_Opacity-3128.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3128.png,Lung_Opacity/masks/Lung_Opacity-3128.png,True,False,299,299,L,153.96,0df72e7a377217860ee24ff681d35919
Lung_Opacity-3129.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3129.png,Lung_Opacity/images/Lung_Opacity-3129.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3129.png,Lung_Opacity/masks/Lung_Opacity-3129.png,True,False,299,299,L,159.43,251049fc6f60edcb0b65c8d5af1d3c30
Lung_Opacity-313.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-313.png,Lung_Opacity/images/Lung_Opacity-313.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-313.png,Lung_Opacity/masks/Lung_Opacity-313.png,True,False,299,299,L,112.64,e030d3bcf9e4c4955656fd0c69f97bc0
Lung_Opacity-3130.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3130.png,Lung_Opacity/images/Lung_Opacity-3130.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3130.png,Lung_Opacity/masks/Lung_Opacity-3130.png,True,False,299,299,L,144.54,6724e042e11f1e3c87a9ebe4587f2b81
Lung_Opacity-3131.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3131.png,Lung_Opacity/images/Lung_Opacity-3131.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3131.png,Lung_Opacity/masks/Lung_Opacity-3131.png,True,False,299,299,L,123.23,e74e0698c07aba50143ef36860c79e21
Lung_Opacity-3132.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3132.png,Lung_Opacity/images/Lung_Opacity-3132.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3132.png,Lung_Opacity/masks/Lung_Opacity-3132.png,True,False,299,299,L,165.43,a4e2fc2a7733b6675a34354e15678f2b
Lung_Opacity-3133.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3133.png,Lung_Opacity/images/Lung_Opacity-3133.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3133.png,Lung_Opacity/masks/Lung_Opacity-3133.png,True,False,299,299,L,109.96,d5dba3f993952de0a1b0742bae075449
Lung_Opacity-3134.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3134.png,Lung_Opacity/images/Lung_Opacity-3134.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3134.png,Lung_Opacity/masks/Lung_Opacity-3134.png,True,False,299,299,L,102.53,9f9df5a20d13194aee0ca44f53c34a6d
Lung_Opacity-3135.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3135.png,Lung_Opacity/images/Lung_Opacity-3135.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3135.png,Lung_Opacity/masks/Lung_Opacity-3135.png,True,False,299,299,L,104.11,37b9c4707c518ebc16cf6e0e4003a373
Lung_Opacity-3136.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3136.png,Lung_Opacity/images/Lung_Opacity-3136.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3136.png,Lung_Opacity/masks/Lung_Opacity-3136.png,True,False,299,299,L,105.01,d28c2195eb47f077c9a2e237990ce818
Lung_Opacity-3137.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3137.png,Lung_Opacity/images/Lung_Opacity-3137.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3137.png,Lung_Opacity/masks/Lung_Opacity-3137.png,True,False,299,299,L,116.89,11c408a9ccdc899475309211e57e3926
Lung_Opacity-3138.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3138.png,Lung_Opacity/images/Lung_Opacity-3138.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3138.png,Lung_Opacity/masks/Lung_Opacity-3138.png,True,False,299,299,L,114.27,2b379ef407253077cdde915e5f176b52
Lung_Opacity-3139.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3139.png,Lung_Opacity/images/Lung_Opacity-3139.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3139.png,Lung_Opacity/masks/Lung_Opacity-3139.png,True,False,299,299,L,114.87,88d2813882fe338fcf88f19f385772f7
Lung_Opacity-314.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-314.png,Lung_Opacity/images/Lung_Opacity-314.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-314.png,Lung_Opacity/masks/Lung_Opacity-314.png,True,False,299,299,L,152.81,3ff675038cd8f64090e2daf3ab6fa9f8
Lung_Opacity-3140.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3140.png,Lung_Opacity/images/Lung_Opacity-3140.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3140.png,Lung_Opacity/masks/Lung_Opacity-3140.png,True,False,299,299,L,91.22,cb64a8f26cd108ccb76c75a1d8f084da
Lung_Opacity-3141.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3141.png,Lung_Opacity/images/Lung_Opacity-3141.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3141.png,Lung_Opacity/masks/Lung_Opacity-3141.png,True,False,299,299,L,117.55,2609e20fb17f62068813d9187645edff
Lung_Opacity-3142.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3142.png,Lung_Opacity/images/Lung_Opacity-3142.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3142.png,Lung_Opacity/masks/Lung_Opacity-3142.png,True,False,299,299,L,114.31,b1f44859c22203bf3c604b3d088c552c
Lung_Opacity-3143.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3143.png,Lung_Opacity/images/Lung_Opacity-3143.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3143.png,Lung_Opacity/masks/Lung_Opacity-3143.png,True,False,299,299,L,98.71,6f04426fb373c69a7e10f39b58099d68
Lung_Opacity-3144.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3144.png,Lung_Opacity/images/Lung_Opacity-3144.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3144.png,Lung_Opacity/masks/Lung_Opacity-3144.png,True,False,299,299,L,78.22,72685110c122510ce7066c25ee33e60c
Lung_Opacity-3145.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3145.png,Lung_Opacity/images/Lung_Opacity-3145.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3145.png,Lung_Opacity/masks/Lung_Opacity-3145.png,True,False,299,299,L,167.04,60d182c5c4824217eb9f912e97948b9e
Lung_Opacity-3146.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3146.png,Lung_Opacity/images/Lung_Opacity-3146.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3146.png,Lung_Opacity/masks/Lung_Opacity-3146.png,True,False,299,299,L,172.82,7d9a6474f3857e51c59590c53f21f031
Lung_Opacity-3147.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3147.png,Lung_Opacity/images/Lung_Opacity-3147.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3147.png,Lung_Opacity/masks/Lung_Opacity-3147.png,True,False,299,299,L,130.87,66d7d8c6261e2bfb73a7d4d05553d5d7
Lung_Opacity-3148.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3148.png,Lung_Opacity/images/Lung_Opacity-3148.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3148.png,Lung_Opacity/masks/Lung_Opacity-3148.png,True,False,299,299,L,168.32,52f80be418e8d08a5b3d5fe2f53ba1ca
Lung_Opacity-3149.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3149.png,Lung_Opacity/images/Lung_Opacity-3149.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3149.png,Lung_Opacity/masks/Lung_Opacity-3149.png,True,False,299,299,L,100.25,df4c57d7ecece71a67b6cedfdb5d7045
Lung_Opacity-315.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-315.png,Lung_Opacity/images/Lung_Opacity-315.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-315.png,Lung_Opacity/masks/Lung_Opacity-315.png,True,False,299,299,L,128.84,68130e1e598fe1c4734592ab81b4117a
Lung_Opacity-3150.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3150.png,Lung_Opacity/images/Lung_Opacity-3150.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3150.png,Lung_Opacity/masks/Lung_Opacity-3150.png,True,False,299,299,L,98.79,c30470cce2f5baf95dec6738aa69aafc
Lung_Opacity-3151.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3151.png,Lung_Opacity/images/Lung_Opacity-3151.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3151.png,Lung_Opacity/masks/Lung_Opacity-3151.png,True,False,299,299,L,124.91,279e271c7a211da2b2af52253c502132
Lung_Opacity-3152.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3152.png,Lung_Opacity/images/Lung_Opacity-3152.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3152.png,Lung_Opacity/masks/Lung_Opacity-3152.png,True,False,299,299,L,104.17,374948e8363e05b73699c33a01c49f7b
Lung_Opacity-3153.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3153.png,Lung_Opacity/images/Lung_Opacity-3153.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3153.png,Lung_Opacity/masks/Lung_Opacity-3153.png,True,False,299,299,L,149.67,b1f8e8f357b0020ef5d9986ea3db1766
Lung_Opacity-3154.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3154.png,Lung_Opacity/images/Lung_Opacity-3154.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3154.png,Lung_Opacity/masks/Lung_Opacity-3154.png,True,False,299,299,L,135.35,6a6984daa60906458cc6ec2e21fbfde5
Lung_Opacity-3155.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3155.png,Lung_Opacity/images/Lung_Opacity-3155.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3155.png,Lung_Opacity/masks/Lung_Opacity-3155.png,True,False,299,299,L,107.58,f7de24be24f4ca42a1d6c0f8bb7258ee
Lung_Opacity-3156.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3156.png,Lung_Opacity/images/Lung_Opacity-3156.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3156.png,Lung_Opacity/masks/Lung_Opacity-3156.png,True,False,299,299,L,106.91,d2d2a5c6324ed36a3a1fcb2e3fe1bfc7
Lung_Opacity-3157.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3157.png,Lung_Opacity/images/Lung_Opacity-3157.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3157.png,Lung_Opacity/masks/Lung_Opacity-3157.png,True,False,299,299,L,111.34,c0c15823ffc00e676747e184bcc46577
Lung_Opacity-3158.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3158.png,Lung_Opacity/images/Lung_Opacity-3158.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3158.png,Lung_Opacity/masks/Lung_Opacity-3158.png,True,False,299,299,L,170.75,2a0af5ff49e72ed045f22a7d1ce5ce42
Lung_Opacity-3159.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3159.png,Lung_Opacity/images/Lung_Opacity-3159.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3159.png,Lung_Opacity/masks/Lung_Opacity-3159.png,True,False,299,299,L,114.65,019eae3c4b3d5859ff0385ed7eed2e15
Lung_Opacity-316.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-316.png,Lung_Opacity/images/Lung_Opacity-316.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-316.png,Lung_Opacity/masks/Lung_Opacity-316.png,True,False,299,299,L,119.77,439774afb898361356fe30ba69c1ad18
Lung_Opacity-3160.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3160.png,Lung_Opacity/images/Lung_Opacity-3160.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3160.png,Lung_Opacity/masks/Lung_Opacity-3160.png,True,False,299,299,L,104.98,ee6a086366828705553ded393c29c950
Lung_Opacity-3161.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3161.png,Lung_Opacity/images/Lung_Opacity-3161.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3161.png,Lung_Opacity/masks/Lung_Opacity-3161.png,True,False,299,299,L,141.21,d870c52405a29c279e503c6be80e92d8
Lung_Opacity-3162.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3162.png,Lung_Opacity/images/Lung_Opacity-3162.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3162.png,Lung_Opacity/masks/Lung_Opacity-3162.png,True,False,299,299,L,109.7,8713105e9e3057b993f32593318d9adb
Lung_Opacity-3163.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3163.png,Lung_Opacity/images/Lung_Opacity-3163.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3163.png,Lung_Opacity/masks/Lung_Opacity-3163.png,True,False,299,299,L,109.29,45b51553804bade8134f3114c543765f
Lung_Opacity-3164.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3164.png,Lung_Opacity/images/Lung_Opacity-3164.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3164.png,Lung_Opacity/masks/Lung_Opacity-3164.png,True,False,299,299,L,164.79,fb7ae1d15ff8ef5064d4e2f8b044c8c7
Lung_Opacity-3165.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3165.png,Lung_Opacity/images/Lung_Opacity-3165.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3165.png,Lung_Opacity/masks/Lung_Opacity-3165.png,True,False,299,299,L,106.28,b83a0831dce6c0f636bbba2f042631b6
Lung_Opacity-3166.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3166.png,Lung_Opacity/images/Lung_Opacity-3166.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3166.png,Lung_Opacity/masks/Lung_Opacity-3166.png,True,False,299,299,L,114.06,6920c1bd604d9457869a599016c671be
Lung_Opacity-3167.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3167.png,Lung_Opacity/images/Lung_Opacity-3167.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3167.png,Lung_Opacity/masks/Lung_Opacity-3167.png,True,False,299,299,L,150.66,41d0cda76bbfa2447d8f08eb2287c14b
Lung_Opacity-3168.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3168.png,Lung_Opacity/images/Lung_Opacity-3168.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3168.png,Lung_Opacity/masks/Lung_Opacity-3168.png,True,False,299,299,L,126.21,7cb9cb6964f7129c81b666b13c40485f
Lung_Opacity-3169.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3169.png,Lung_Opacity/images/Lung_Opacity-3169.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3169.png,Lung_Opacity/masks/Lung_Opacity-3169.png,True,False,299,299,L,147.23,9184fbc5cb3277ea83c333f319100650
Lung_Opacity-317.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-317.png,Lung_Opacity/images/Lung_Opacity-317.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-317.png,Lung_Opacity/masks/Lung_Opacity-317.png,True,False,299,299,L,108.25,63014c945bcdf18a8972e8bd1f018d74
Lung_Opacity-3170.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3170.png,Lung_Opacity/images/Lung_Opacity-3170.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3170.png,Lung_Opacity/masks/Lung_Opacity-3170.png,True,False,299,299,L,167.94,503434834ce048978fe30a84a59b6e20
Lung_Opacity-3171.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3171.png,Lung_Opacity/images/Lung_Opacity-3171.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3171.png,Lung_Opacity/masks/Lung_Opacity-3171.png,True,False,299,299,L,121.64,570248e4ec53971f90847257a6d0b91e
Lung_Opacity-3172.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3172.png,Lung_Opacity/images/Lung_Opacity-3172.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3172.png,Lung_Opacity/masks/Lung_Opacity-3172.png,True,False,299,299,L,115.44,e5319a10b227eeeaba2787d6824a9ddb
Lung_Opacity-3173.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3173.png,Lung_Opacity/images/Lung_Opacity-3173.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3173.png,Lung_Opacity/masks/Lung_Opacity-3173.png,True,False,299,299,L,70.18,581eac2040829336ea49503007b7d918
Lung_Opacity-3174.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3174.png,Lung_Opacity/images/Lung_Opacity-3174.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3174.png,Lung_Opacity/masks/Lung_Opacity-3174.png,True,False,299,299,L,112.52,c208072d2dd316270b3ca501a8ab02eb
Lung_Opacity-3175.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3175.png,Lung_Opacity/images/Lung_Opacity-3175.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3175.png,Lung_Opacity/masks/Lung_Opacity-3175.png,True,False,299,299,L,154.25,75ebaffd9d53af71e010d0333599e26e
Lung_Opacity-3176.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3176.png,Lung_Opacity/images/Lung_Opacity-3176.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3176.png,Lung_Opacity/masks/Lung_Opacity-3176.png,True,False,299,299,L,89.01,6a25ba6e2c6f9a76699299e7af3183bf
Lung_Opacity-3177.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3177.png,Lung_Opacity/images/Lung_Opacity-3177.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3177.png,Lung_Opacity/masks/Lung_Opacity-3177.png,True,False,299,299,L,113.14,4656793d26b77b67bd930967c92444a3
Lung_Opacity-3178.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3178.png,Lung_Opacity/images/Lung_Opacity-3178.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3178.png,Lung_Opacity/masks/Lung_Opacity-3178.png,True,False,299,299,L,120.29,1f343054cd2b4409c18876c5e72697a4
Lung_Opacity-3179.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3179.png,Lung_Opacity/images/Lung_Opacity-3179.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3179.png,Lung_Opacity/masks/Lung_Opacity-3179.png,True,False,299,299,L,114.91,f4eb1892df59b3a51a41c2844ebfb7f2
Lung_Opacity-318.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-318.png,Lung_Opacity/images/Lung_Opacity-318.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-318.png,Lung_Opacity/masks/Lung_Opacity-318.png,True,False,299,299,L,165.91,2359aad77cde8961af57d63dbecfa920
Lung_Opacity-3180.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3180.png,Lung_Opacity/images/Lung_Opacity-3180.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3180.png,Lung_Opacity/masks/Lung_Opacity-3180.png,True,False,299,299,L,144.96,afd29dfefd6f341b8f2f022c06c40735
Lung_Opacity-3181.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3181.png,Lung_Opacity/images/Lung_Opacity-3181.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3181.png,Lung_Opacity/masks/Lung_Opacity-3181.png,True,False,299,299,L,109.5,8c33930dd2a8677a13f33cca769447fc
Lung_Opacity-3182.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3182.png,Lung_Opacity/images/Lung_Opacity-3182.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3182.png,Lung_Opacity/masks/Lung_Opacity-3182.png,True,False,299,299,L,106.35,f006a42a71ae1d55d0e8e9bb00e72711
Lung_Opacity-3183.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3183.png,Lung_Opacity/images/Lung_Opacity-3183.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3183.png,Lung_Opacity/masks/Lung_Opacity-3183.png,True,False,299,299,L,80.58,95e653869922804db3860897b0a2254c
Lung_Opacity-3184.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3184.png,Lung_Opacity/images/Lung_Opacity-3184.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3184.png,Lung_Opacity/masks/Lung_Opacity-3184.png,True,False,299,299,L,125.53,ec5e2969d4ffd978e1d696bae0e388f6
Lung_Opacity-3185.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3185.png,Lung_Opacity/images/Lung_Opacity-3185.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3185.png,Lung_Opacity/masks/Lung_Opacity-3185.png,True,False,299,299,L,117.69,ec9d69f6158dc85404491187f89b24ad
Lung_Opacity-3186.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3186.png,Lung_Opacity/images/Lung_Opacity-3186.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3186.png,Lung_Opacity/masks/Lung_Opacity-3186.png,True,False,299,299,L,112.94,8efa54783f036ba9661a91efc01cccda
Lung_Opacity-3187.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3187.png,Lung_Opacity/images/Lung_Opacity-3187.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3187.png,Lung_Opacity/masks/Lung_Opacity-3187.png,True,False,299,299,L,120.42,6af2e0255e4f422bde3298a31a9cc8a0
Lung_Opacity-3188.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3188.png,Lung_Opacity/images/Lung_Opacity-3188.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3188.png,Lung_Opacity/masks/Lung_Opacity-3188.png,True,False,299,299,L,113.39,85adbe1e5cba5d855686ed01658b28b9
Lung_Opacity-3189.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3189.png,Lung_Opacity/images/Lung_Opacity-3189.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3189.png,Lung_Opacity/masks/Lung_Opacity-3189.png,True,False,299,299,L,107.81,2e306e74969ab9025b2cf01a7f483e1c
Lung_Opacity-319.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-319.png,Lung_Opacity/images/Lung_Opacity-319.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-319.png,Lung_Opacity/masks/Lung_Opacity-319.png,True,False,299,299,L,158.33,afa88be1dd3a9df9adcd27b9a5cc0d18
Lung_Opacity-3190.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3190.png,Lung_Opacity/images/Lung_Opacity-3190.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3190.png,Lung_Opacity/masks/Lung_Opacity-3190.png,True,False,299,299,L,151.77,1a806fa086472fbd0c198d296fa43fc2
Lung_Opacity-3191.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3191.png,Lung_Opacity/images/Lung_Opacity-3191.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3191.png,Lung_Opacity/masks/Lung_Opacity-3191.png,True,False,299,299,L,148.18,e92d0cbfd2b71493af0bab354072f9f7
Lung_Opacity-3192.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3192.png,Lung_Opacity/images/Lung_Opacity-3192.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3192.png,Lung_Opacity/masks/Lung_Opacity-3192.png,True,False,299,299,L,118.91,fcdb4ebe90fa2edbb078e33646ab90be
Lung_Opacity-3193.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3193.png,Lung_Opacity/images/Lung_Opacity-3193.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3193.png,Lung_Opacity/masks/Lung_Opacity-3193.png,True,False,299,299,L,133.51,0a84f01df2e315d2ec5dcf2d98b84093
Lung_Opacity-3194.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3194.png,Lung_Opacity/images/Lung_Opacity-3194.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3194.png,Lung_Opacity/masks/Lung_Opacity-3194.png,True,False,299,299,L,112.65,692ac2440d5b77bd381094160e6c8ac5
Lung_Opacity-3195.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3195.png,Lung_Opacity/images/Lung_Opacity-3195.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3195.png,Lung_Opacity/masks/Lung_Opacity-3195.png,True,False,299,299,L,103.71,31de7494149c610f4d80f54c04a2a768
Lung_Opacity-3196.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3196.png,Lung_Opacity/images/Lung_Opacity-3196.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3196.png,Lung_Opacity/masks/Lung_Opacity-3196.png,True,False,299,299,L,111.08,21014b527b1112feb3ec04bc6863e384
Lung_Opacity-3197.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3197.png,Lung_Opacity/images/Lung_Opacity-3197.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3197.png,Lung_Opacity/masks/Lung_Opacity-3197.png,True,False,299,299,L,143.39,f6227f1d81757268ebd92ab2e6e8d0f2
Lung_Opacity-3198.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3198.png,Lung_Opacity/images/Lung_Opacity-3198.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3198.png,Lung_Opacity/masks/Lung_Opacity-3198.png,True,False,299,299,L,168.56,38314e278ba3acbc190cd29b9edfd79d
Lung_Opacity-3199.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3199.png,Lung_Opacity/images/Lung_Opacity-3199.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3199.png,Lung_Opacity/masks/Lung_Opacity-3199.png,True,False,299,299,L,101.43,d04f9d05a47632083887b797a12bc0b3
Lung_Opacity-32.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-32.png,Lung_Opacity/images/Lung_Opacity-32.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-32.png,Lung_Opacity/masks/Lung_Opacity-32.png,True,False,299,299,L,145.99,caa0f57f2fb34bc513d3bcb7a10a0c46
Lung_Opacity-320.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-320.png,Lung_Opacity/images/Lung_Opacity-320.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-320.png,Lung_Opacity/masks/Lung_Opacity-320.png,True,False,299,299,L,131.39,0a28d9d503f40a96873380745cfea936
Lung_Opacity-3200.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3200.png,Lung_Opacity/images/Lung_Opacity-3200.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3200.png,Lung_Opacity/masks/Lung_Opacity-3200.png,True,False,299,299,L,112.86,8c4fbeef9843839e0fcf26d7ace7af87
Lung_Opacity-3201.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3201.png,Lung_Opacity/images/Lung_Opacity-3201.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3201.png,Lung_Opacity/masks/Lung_Opacity-3201.png,True,False,299,299,L,114.81,d325ac898dfdcf55c62fe1b44273403f
Lung_Opacity-3202.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3202.png,Lung_Opacity/images/Lung_Opacity-3202.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3202.png,Lung_Opacity/masks/Lung_Opacity-3202.png,True,False,299,299,L,126.07,24d634705eb1cbf3aabbd036df4ad4f4
Lung_Opacity-3203.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3203.png,Lung_Opacity/images/Lung_Opacity-3203.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3203.png,Lung_Opacity/masks/Lung_Opacity-3203.png,True,False,299,299,L,124.15,de812a390083db7a86821dd9563a2095
Lung_Opacity-3204.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3204.png,Lung_Opacity/images/Lung_Opacity-3204.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3204.png,Lung_Opacity/masks/Lung_Opacity-3204.png,True,False,299,299,L,176.5,ecb16c2d842bf0861e74ea2d7274ca60
Lung_Opacity-3205.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3205.png,Lung_Opacity/images/Lung_Opacity-3205.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3205.png,Lung_Opacity/masks/Lung_Opacity-3205.png,True,False,299,299,L,160.13,913d1a0fe342246da06a1fe8fafe4010
Lung_Opacity-3206.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3206.png,Lung_Opacity/images/Lung_Opacity-3206.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3206.png,Lung_Opacity/masks/Lung_Opacity-3206.png,True,False,299,299,L,125.32,564a8c75b0c9f5e38555fffd5c9481c4
Lung_Opacity-3207.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3207.png,Lung_Opacity/images/Lung_Opacity-3207.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3207.png,Lung_Opacity/masks/Lung_Opacity-3207.png,True,False,299,299,L,98.1,90165fe9a2f4e4bc7552cd9cdcfcc87f
Lung_Opacity-3208.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3208.png,Lung_Opacity/images/Lung_Opacity-3208.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3208.png,Lung_Opacity/masks/Lung_Opacity-3208.png,True,False,299,299,L,101.41,bf06dd20cf9d76bf40c44c38cae13f7e
Lung_Opacity-3209.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3209.png,Lung_Opacity/images/Lung_Opacity-3209.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3209.png,Lung_Opacity/masks/Lung_Opacity-3209.png,True,False,299,299,L,156.84,b8c57609c5780b31ff67d4a3f3caa0ff
Lung_Opacity-321.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-321.png,Lung_Opacity/images/Lung_Opacity-321.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-321.png,Lung_Opacity/masks/Lung_Opacity-321.png,True,False,299,299,L,117.67,c72488432e7c47d137d0861bca2f5d30
Lung_Opacity-3210.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3210.png,Lung_Opacity/images/Lung_Opacity-3210.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3210.png,Lung_Opacity/masks/Lung_Opacity-3210.png,True,False,299,299,L,111.12,8e453bf15c2de45f2696eecd64341bef
Lung_Opacity-3211.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3211.png,Lung_Opacity/images/Lung_Opacity-3211.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3211.png,Lung_Opacity/masks/Lung_Opacity-3211.png,True,False,299,299,L,106.58,a7ebb5bedb8876cab3abc2db7957e2af
Lung_Opacity-3212.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3212.png,Lung_Opacity/images/Lung_Opacity-3212.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3212.png,Lung_Opacity/masks/Lung_Opacity-3212.png,True,False,299,299,L,128.1,2d691319da06eda5b931ae637e5851b3
Lung_Opacity-3213.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3213.png,Lung_Opacity/images/Lung_Opacity-3213.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3213.png,Lung_Opacity/masks/Lung_Opacity-3213.png,True,False,299,299,L,110.55,1251733ce0686510f3a3e10fa5498ee2
Lung_Opacity-3214.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3214.png,Lung_Opacity/images/Lung_Opacity-3214.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3214.png,Lung_Opacity/masks/Lung_Opacity-3214.png,True,False,299,299,L,125.38,2f805cfd4806901a685ecf6ea4ad0d65
Lung_Opacity-3215.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3215.png,Lung_Opacity/images/Lung_Opacity-3215.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3215.png,Lung_Opacity/masks/Lung_Opacity-3215.png,True,False,299,299,L,144.26,72738e0a0e1ff17c2740bd87796715f9
Lung_Opacity-3216.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3216.png,Lung_Opacity/images/Lung_Opacity-3216.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3216.png,Lung_Opacity/masks/Lung_Opacity-3216.png,True,False,299,299,L,120.44,6146de0bf61f252b2c06d4efeaf281dc
Lung_Opacity-3217.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3217.png,Lung_Opacity/images/Lung_Opacity-3217.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3217.png,Lung_Opacity/masks/Lung_Opacity-3217.png,True,False,299,299,L,181.74,60d6b7851385926ff463c72d2e577a99
Lung_Opacity-3218.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3218.png,Lung_Opacity/images/Lung_Opacity-3218.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3218.png,Lung_Opacity/masks/Lung_Opacity-3218.png,True,False,299,299,L,124.01,fb323acd55d6101463e3e987aef108f2
Lung_Opacity-3219.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3219.png,Lung_Opacity/images/Lung_Opacity-3219.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3219.png,Lung_Opacity/masks/Lung_Opacity-3219.png,True,False,299,299,L,138.85,29844a91a9e55cd62284fcafd7adac74
Lung_Opacity-322.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-322.png,Lung_Opacity/images/Lung_Opacity-322.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-322.png,Lung_Opacity/masks/Lung_Opacity-322.png,True,False,299,299,L,155.42,6f0a43e0ffc70c20759d0ccb05f41b77
Lung_Opacity-3220.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3220.png,Lung_Opacity/images/Lung_Opacity-3220.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3220.png,Lung_Opacity/masks/Lung_Opacity-3220.png,True,False,299,299,L,118.93,7f9ec6b75da356bb150b632502a58046
Lung_Opacity-3221.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3221.png,Lung_Opacity/images/Lung_Opacity-3221.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3221.png,Lung_Opacity/masks/Lung_Opacity-3221.png,True,False,299,299,L,120.89,ad802be46bf262cc851bdb54964e1373
Lung_Opacity-3222.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3222.png,Lung_Opacity/images/Lung_Opacity-3222.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3222.png,Lung_Opacity/masks/Lung_Opacity-3222.png,True,False,299,299,L,144.47,1fd7a9d49f4bd1592243c3e02fb834bd
Lung_Opacity-3223.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3223.png,Lung_Opacity/images/Lung_Opacity-3223.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3223.png,Lung_Opacity/masks/Lung_Opacity-3223.png,True,False,299,299,L,170.72,300a92927b8e21b08359b1bd1f2116ee
Lung_Opacity-3224.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3224.png,Lung_Opacity/images/Lung_Opacity-3224.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3224.png,Lung_Opacity/masks/Lung_Opacity-3224.png,True,False,299,299,L,110.92,ead5a11a0422a16d86f362c252f25b52
Lung_Opacity-3225.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3225.png,Lung_Opacity/images/Lung_Opacity-3225.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3225.png,Lung_Opacity/masks/Lung_Opacity-3225.png,True,False,299,299,L,128.58,65c80e29ab0273650c24d29a26b74868
Lung_Opacity-3226.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3226.png,Lung_Opacity/images/Lung_Opacity-3226.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3226.png,Lung_Opacity/masks/Lung_Opacity-3226.png,True,False,299,299,L,114.78,cca4a4800c860e3dc46d0182fc1b641f
Lung_Opacity-3227.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3227.png,Lung_Opacity/images/Lung_Opacity-3227.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3227.png,Lung_Opacity/masks/Lung_Opacity-3227.png,True,False,299,299,L,109.14,0d23d56075fcfd88c135952e75fc803d
Lung_Opacity-3228.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3228.png,Lung_Opacity/images/Lung_Opacity-3228.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3228.png,Lung_Opacity/masks/Lung_Opacity-3228.png,True,False,299,299,L,160.08,23ca3ad134515c396a9d9e4d83c33245
Lung_Opacity-3229.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3229.png,Lung_Opacity/images/Lung_Opacity-3229.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3229.png,Lung_Opacity/masks/Lung_Opacity-3229.png,True,False,299,299,L,158.55,3bcf4f7952f4b161a37733c49efc2e26
Lung_Opacity-323.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-323.png,Lung_Opacity/images/Lung_Opacity-323.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-323.png,Lung_Opacity/masks/Lung_Opacity-323.png,True,False,299,299,L,149.49,e1697e0ddaf2d44b344ba0c53e589915
Lung_Opacity-3230.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3230.png,Lung_Opacity/images/Lung_Opacity-3230.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3230.png,Lung_Opacity/masks/Lung_Opacity-3230.png,True,False,299,299,L,115.03,a9a3bf5b8c1f83d7944305e3006f242b
Lung_Opacity-3231.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3231.png,Lung_Opacity/images/Lung_Opacity-3231.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3231.png,Lung_Opacity/masks/Lung_Opacity-3231.png,True,False,299,299,L,164.6,c94ffff6d995e7c7425c892e52d8811d
Lung_Opacity-3232.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3232.png,Lung_Opacity/images/Lung_Opacity-3232.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3232.png,Lung_Opacity/masks/Lung_Opacity-3232.png,True,False,299,299,L,110.18,feb73c8f55f4967037ac64f8c5016a60
Lung_Opacity-3233.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3233.png,Lung_Opacity/images/Lung_Opacity-3233.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3233.png,Lung_Opacity/masks/Lung_Opacity-3233.png,True,False,299,299,L,115.09,a62412ff7e0b3c46c6be8aab064a908a
Lung_Opacity-3234.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3234.png,Lung_Opacity/images/Lung_Opacity-3234.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3234.png,Lung_Opacity/masks/Lung_Opacity-3234.png,True,False,299,299,L,104.4,09f5c220b07ef39f75e20ce440a0f078
Lung_Opacity-3235.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3235.png,Lung_Opacity/images/Lung_Opacity-3235.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3235.png,Lung_Opacity/masks/Lung_Opacity-3235.png,True,False,299,299,L,118.4,da63237ed23dcad3899ad3298a2270e4
Lung_Opacity-3236.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3236.png,Lung_Opacity/images/Lung_Opacity-3236.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3236.png,Lung_Opacity/masks/Lung_Opacity-3236.png,True,False,299,299,L,137.51,27e6c5a7df4a66db1c692b0513f6ee1d
Lung_Opacity-3237.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3237.png,Lung_Opacity/images/Lung_Opacity-3237.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3237.png,Lung_Opacity/masks/Lung_Opacity-3237.png,True,False,299,299,L,120.76,66a0094cb4f690243f2e2ee250992f3a
Lung_Opacity-3238.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3238.png,Lung_Opacity/images/Lung_Opacity-3238.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3238.png,Lung_Opacity/masks/Lung_Opacity-3238.png,True,False,299,299,L,109.83,7e603880343baff5a9d95c370bc74769
Lung_Opacity-3239.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3239.png,Lung_Opacity/images/Lung_Opacity-3239.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3239.png,Lung_Opacity/masks/Lung_Opacity-3239.png,True,False,299,299,L,158.16,ef2b1a708a61643b4016222a2f77991a
Lung_Opacity-324.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-324.png,Lung_Opacity/images/Lung_Opacity-324.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-324.png,Lung_Opacity/masks/Lung_Opacity-324.png,True,False,299,299,L,115.23,ca705760af0c98347a3143fe88b36b7e
Lung_Opacity-3240.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3240.png,Lung_Opacity/images/Lung_Opacity-3240.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3240.png,Lung_Opacity/masks/Lung_Opacity-3240.png,True,False,299,299,L,128.01,e5dfe6284ed7026837c16504d3aaa032
Lung_Opacity-3241.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3241.png,Lung_Opacity/images/Lung_Opacity-3241.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3241.png,Lung_Opacity/masks/Lung_Opacity-3241.png,True,False,299,299,L,138.08,94bbdad16311a1096e8f39edbc28bef8
Lung_Opacity-3242.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3242.png,Lung_Opacity/images/Lung_Opacity-3242.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3242.png,Lung_Opacity/masks/Lung_Opacity-3242.png,True,False,299,299,L,175.3,bb576afdbd1371104333608b582fec0a
Lung_Opacity-3243.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3243.png,Lung_Opacity/images/Lung_Opacity-3243.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3243.png,Lung_Opacity/masks/Lung_Opacity-3243.png,True,False,299,299,L,116.98,c5552d191c6f11b7b9217751b6e9ea5e
Lung_Opacity-3244.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3244.png,Lung_Opacity/images/Lung_Opacity-3244.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3244.png,Lung_Opacity/masks/Lung_Opacity-3244.png,True,False,299,299,L,117.41,76cac96adfec73afbcacb2d5333906db
Lung_Opacity-3245.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3245.png,Lung_Opacity/images/Lung_Opacity-3245.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3245.png,Lung_Opacity/masks/Lung_Opacity-3245.png,True,False,299,299,L,159.12,5adcc6185399721164afd25e0d8e1f65
Lung_Opacity-3246.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3246.png,Lung_Opacity/images/Lung_Opacity-3246.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3246.png,Lung_Opacity/masks/Lung_Opacity-3246.png,True,False,299,299,L,115.11,bcd17391bc7c228e8907ade80184cf55
Lung_Opacity-3247.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3247.png,Lung_Opacity/images/Lung_Opacity-3247.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3247.png,Lung_Opacity/masks/Lung_Opacity-3247.png,True,False,299,299,L,76.79,4529f8e3e5eb5fd83edfcbba493498fa
Lung_Opacity-3248.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3248.png,Lung_Opacity/images/Lung_Opacity-3248.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3248.png,Lung_Opacity/masks/Lung_Opacity-3248.png,True,False,299,299,L,67.01,994e411fb66253c60bfc45c6496c9d09
Lung_Opacity-3249.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3249.png,Lung_Opacity/images/Lung_Opacity-3249.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3249.png,Lung_Opacity/masks/Lung_Opacity-3249.png,True,False,299,299,L,163.53,b24e2a970a950e1c2d665fde48f7d002
Lung_Opacity-325.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-325.png,Lung_Opacity/images/Lung_Opacity-325.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-325.png,Lung_Opacity/masks/Lung_Opacity-325.png,True,False,299,299,L,123.45,6f91b50db97298e4a8d3b519dc69eccb
Lung_Opacity-3250.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3250.png,Lung_Opacity/images/Lung_Opacity-3250.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3250.png,Lung_Opacity/masks/Lung_Opacity-3250.png,True,False,299,299,L,105.66,878e7916d27419f24bd9eb7df26e5bc6
Lung_Opacity-3251.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3251.png,Lung_Opacity/images/Lung_Opacity-3251.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3251.png,Lung_Opacity/masks/Lung_Opacity-3251.png,True,False,299,299,L,157.51,009aa51460e177858e653b1905300f41
Lung_Opacity-3252.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3252.png,Lung_Opacity/images/Lung_Opacity-3252.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3252.png,Lung_Opacity/masks/Lung_Opacity-3252.png,True,False,299,299,L,176.69,0b4ce1b0ad858be963b9caafeba23d6e
Lung_Opacity-3253.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3253.png,Lung_Opacity/images/Lung_Opacity-3253.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3253.png,Lung_Opacity/masks/Lung_Opacity-3253.png,True,False,299,299,L,105.75,3ee07d42f04e1cf7547e262bb5bb8875
Lung_Opacity-3254.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3254.png,Lung_Opacity/images/Lung_Opacity-3254.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3254.png,Lung_Opacity/masks/Lung_Opacity-3254.png,True,False,299,299,L,117.9,16d3b1ba1313c63361db625812db04bd
Lung_Opacity-3255.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3255.png,Lung_Opacity/images/Lung_Opacity-3255.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3255.png,Lung_Opacity/masks/Lung_Opacity-3255.png,True,False,299,299,L,163.78,38753585dfbc0d391a889a4b7e04b688
Lung_Opacity-3256.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3256.png,Lung_Opacity/images/Lung_Opacity-3256.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3256.png,Lung_Opacity/masks/Lung_Opacity-3256.png,True,False,299,299,L,142.76,febd3b81e2fa0ca4dc2ac64dd6955bec
Lung_Opacity-3257.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3257.png,Lung_Opacity/images/Lung_Opacity-3257.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3257.png,Lung_Opacity/masks/Lung_Opacity-3257.png,True,False,299,299,L,138.89,8698da347f1ec9d847206ebbc788957a
Lung_Opacity-3258.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3258.png,Lung_Opacity/images/Lung_Opacity-3258.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3258.png,Lung_Opacity/masks/Lung_Opacity-3258.png,True,False,299,299,L,167.42,3944a81288749068d4cef88ed9a67bb9
Lung_Opacity-3259.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3259.png,Lung_Opacity/images/Lung_Opacity-3259.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3259.png,Lung_Opacity/masks/Lung_Opacity-3259.png,True,False,299,299,L,137.21,2952c7553643624433bf450e5169eff9
Lung_Opacity-326.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-326.png,Lung_Opacity/images/Lung_Opacity-326.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-326.png,Lung_Opacity/masks/Lung_Opacity-326.png,True,False,299,299,L,124.96,4d9701e075fc8428cdfb92b98e17ddb2
Lung_Opacity-3260.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3260.png,Lung_Opacity/images/Lung_Opacity-3260.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3260.png,Lung_Opacity/masks/Lung_Opacity-3260.png,True,False,299,299,L,95.04,f967f91982b34dcd304021c7a298885b
Lung_Opacity-3261.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3261.png,Lung_Opacity/images/Lung_Opacity-3261.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3261.png,Lung_Opacity/masks/Lung_Opacity-3261.png,True,False,299,299,L,90.12,b701cb180fe73a22447c6b7bf622b08a
Lung_Opacity-3262.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3262.png,Lung_Opacity/images/Lung_Opacity-3262.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3262.png,Lung_Opacity/masks/Lung_Opacity-3262.png,True,False,299,299,L,113.31,45a41339593590fcb0459fa94c8b1481
Lung_Opacity-3263.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3263.png,Lung_Opacity/images/Lung_Opacity-3263.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3263.png,Lung_Opacity/masks/Lung_Opacity-3263.png,True,False,299,299,L,113.93,cdbe68ff46e3d0b5d17f42a7a621d783
Lung_Opacity-3264.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3264.png,Lung_Opacity/images/Lung_Opacity-3264.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3264.png,Lung_Opacity/masks/Lung_Opacity-3264.png,True,False,299,299,L,141.39,042ef89721530a523533f56a01a8bfff
Lung_Opacity-3265.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3265.png,Lung_Opacity/images/Lung_Opacity-3265.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3265.png,Lung_Opacity/masks/Lung_Opacity-3265.png,True,False,299,299,L,119.06,eb958ea4834873247e0f1d6cb1b7d9e5
Lung_Opacity-3266.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3266.png,Lung_Opacity/images/Lung_Opacity-3266.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3266.png,Lung_Opacity/masks/Lung_Opacity-3266.png,True,False,299,299,L,106.82,58c1c6e5570e612d5cc453f752ad947b
Lung_Opacity-3267.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3267.png,Lung_Opacity/images/Lung_Opacity-3267.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3267.png,Lung_Opacity/masks/Lung_Opacity-3267.png,True,False,299,299,L,104.95,ada27c1309a89b7c1109af8a8de364c3
Lung_Opacity-3268.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3268.png,Lung_Opacity/images/Lung_Opacity-3268.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3268.png,Lung_Opacity/masks/Lung_Opacity-3268.png,True,False,299,299,L,102.79,694ead373a866e0429b3f5223e664b92
Lung_Opacity-3269.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3269.png,Lung_Opacity/images/Lung_Opacity-3269.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3269.png,Lung_Opacity/masks/Lung_Opacity-3269.png,True,False,299,299,L,112.89,6ba2118be133b10d9861d3db05e849bd
Lung_Opacity-327.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-327.png,Lung_Opacity/images/Lung_Opacity-327.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-327.png,Lung_Opacity/masks/Lung_Opacity-327.png,True,False,299,299,L,155.33,2cc29b28977dd17ab530da82e94450f1
Lung_Opacity-3270.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3270.png,Lung_Opacity/images/Lung_Opacity-3270.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3270.png,Lung_Opacity/masks/Lung_Opacity-3270.png,True,False,299,299,L,92.89,181d383c64b200cfab17153cc763741b
Lung_Opacity-3271.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3271.png,Lung_Opacity/images/Lung_Opacity-3271.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3271.png,Lung_Opacity/masks/Lung_Opacity-3271.png,True,False,299,299,L,117.85,a90cc35338189f0f66b945a48aa84983
Lung_Opacity-3272.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3272.png,Lung_Opacity/images/Lung_Opacity-3272.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3272.png,Lung_Opacity/masks/Lung_Opacity-3272.png,True,False,299,299,L,100.69,148807040f7e78f36181c70b3f2afb1a
Lung_Opacity-3273.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3273.png,Lung_Opacity/images/Lung_Opacity-3273.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3273.png,Lung_Opacity/masks/Lung_Opacity-3273.png,True,False,299,299,L,118.08,4439b20b7ed74ac8858b75354b9782f1
Lung_Opacity-3274.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3274.png,Lung_Opacity/images/Lung_Opacity-3274.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3274.png,Lung_Opacity/masks/Lung_Opacity-3274.png,True,False,299,299,L,129.86,15dc812e293f91312fef5cf99ba6b07f
Lung_Opacity-3275.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3275.png,Lung_Opacity/images/Lung_Opacity-3275.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3275.png,Lung_Opacity/masks/Lung_Opacity-3275.png,True,False,299,299,L,146.78,8a36e240a33f19c87cfd090b247866e0
Lung_Opacity-3276.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3276.png,Lung_Opacity/images/Lung_Opacity-3276.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3276.png,Lung_Opacity/masks/Lung_Opacity-3276.png,True,False,299,299,L,113.28,a26ea7167ecacffc87e64da50332fe83
Lung_Opacity-3277.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3277.png,Lung_Opacity/images/Lung_Opacity-3277.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3277.png,Lung_Opacity/masks/Lung_Opacity-3277.png,True,False,299,299,L,122.86,1929e0edd7f51f9ebb30e613e0ab66d9
Lung_Opacity-3278.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3278.png,Lung_Opacity/images/Lung_Opacity-3278.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3278.png,Lung_Opacity/masks/Lung_Opacity-3278.png,True,False,299,299,L,97.57,8325d511a9129a29c5d335c2931d84d5
Lung_Opacity-3279.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3279.png,Lung_Opacity/images/Lung_Opacity-3279.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3279.png,Lung_Opacity/masks/Lung_Opacity-3279.png,True,False,299,299,L,118.24,691711e69e3d90d29a526d6c3dd59cd5
Lung_Opacity-328.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-328.png,Lung_Opacity/images/Lung_Opacity-328.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-328.png,Lung_Opacity/masks/Lung_Opacity-328.png,True,False,299,299,L,93.41,83e7f32e38d79dd0a7563983ae77ecf8
Lung_Opacity-3280.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3280.png,Lung_Opacity/images/Lung_Opacity-3280.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3280.png,Lung_Opacity/masks/Lung_Opacity-3280.png,True,False,299,299,L,109.0,06e208eef7062fdcbd849f68e377d3e0
Lung_Opacity-3281.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3281.png,Lung_Opacity/images/Lung_Opacity-3281.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3281.png,Lung_Opacity/masks/Lung_Opacity-3281.png,True,False,299,299,L,160.44,e8cc87a6c67a511d87e383170a7f7553
Lung_Opacity-3282.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3282.png,Lung_Opacity/images/Lung_Opacity-3282.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3282.png,Lung_Opacity/masks/Lung_Opacity-3282.png,True,False,299,299,L,132.15,393933ed3c776709dc27a145f9783e7e
Lung_Opacity-3283.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3283.png,Lung_Opacity/images/Lung_Opacity-3283.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3283.png,Lung_Opacity/masks/Lung_Opacity-3283.png,True,False,299,299,L,114.64,8e6e3b2dc970a7f5c119c49d5aa9010f
Lung_Opacity-3284.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3284.png,Lung_Opacity/images/Lung_Opacity-3284.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3284.png,Lung_Opacity/masks/Lung_Opacity-3284.png,True,False,299,299,L,135.29,ff45b640711863cbafee097f93857976
Lung_Opacity-3285.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3285.png,Lung_Opacity/images/Lung_Opacity-3285.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3285.png,Lung_Opacity/masks/Lung_Opacity-3285.png,True,False,299,299,L,126.67,2bdfcee42a5ddf4f3977979b152a9c7a
Lung_Opacity-3286.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3286.png,Lung_Opacity/images/Lung_Opacity-3286.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3286.png,Lung_Opacity/masks/Lung_Opacity-3286.png,True,False,299,299,L,126.84,f1088e1282e9a52040a228d6354ec2d1
Lung_Opacity-3287.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3287.png,Lung_Opacity/images/Lung_Opacity-3287.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3287.png,Lung_Opacity/masks/Lung_Opacity-3287.png,True,False,299,299,L,125.91,2f01ab40f281d594eb9ca15072fa3dd8
Lung_Opacity-3288.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3288.png,Lung_Opacity/images/Lung_Opacity-3288.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3288.png,Lung_Opacity/masks/Lung_Opacity-3288.png,True,False,299,299,L,158.94,d4706785a27ea7d3a6c4afec45a1f7c5
Lung_Opacity-3289.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3289.png,Lung_Opacity/images/Lung_Opacity-3289.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3289.png,Lung_Opacity/masks/Lung_Opacity-3289.png,True,False,299,299,L,123.87,c1aa767137c1c52702dceb99a7e4e146
Lung_Opacity-329.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-329.png,Lung_Opacity/images/Lung_Opacity-329.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-329.png,Lung_Opacity/masks/Lung_Opacity-329.png,True,False,299,299,L,108.56,c0a8e99c1c793be358d5cfca26a9b078
Lung_Opacity-3290.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3290.png,Lung_Opacity/images/Lung_Opacity-3290.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3290.png,Lung_Opacity/masks/Lung_Opacity-3290.png,True,False,299,299,L,56.77,5fda4fae5aa2f7de2e036673ffa4efaf
Lung_Opacity-3291.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3291.png,Lung_Opacity/images/Lung_Opacity-3291.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3291.png,Lung_Opacity/masks/Lung_Opacity-3291.png,True,False,299,299,L,119.15,1fd1e0ddb5d6caeedb566176e8999992
Lung_Opacity-3292.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3292.png,Lung_Opacity/images/Lung_Opacity-3292.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3292.png,Lung_Opacity/masks/Lung_Opacity-3292.png,True,False,299,299,L,127.73,3ea22b9d47a54c076ee45299a82b5b8a
Lung_Opacity-3293.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3293.png,Lung_Opacity/images/Lung_Opacity-3293.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3293.png,Lung_Opacity/masks/Lung_Opacity-3293.png,True,False,299,299,L,127.43,fdc77ee66938eafc28ca447352b0e13e
Lung_Opacity-3294.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3294.png,Lung_Opacity/images/Lung_Opacity-3294.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3294.png,Lung_Opacity/masks/Lung_Opacity-3294.png,True,False,299,299,L,109.42,899cc23e48c8568304cd29c76d1df7d0
Lung_Opacity-3295.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3295.png,Lung_Opacity/images/Lung_Opacity-3295.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3295.png,Lung_Opacity/masks/Lung_Opacity-3295.png,True,False,299,299,L,96.99,bc41055b478be9b2d801bb866a01a395
Lung_Opacity-3296.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3296.png,Lung_Opacity/images/Lung_Opacity-3296.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3296.png,Lung_Opacity/masks/Lung_Opacity-3296.png,True,False,299,299,L,166.4,04ab99a65aeb8ef89a5b3100f439adb3
Lung_Opacity-3297.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3297.png,Lung_Opacity/images/Lung_Opacity-3297.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3297.png,Lung_Opacity/masks/Lung_Opacity-3297.png,True,False,299,299,L,106.35,94bf6f7351b6d5ffab4b386e58d51542
Lung_Opacity-3298.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3298.png,Lung_Opacity/images/Lung_Opacity-3298.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3298.png,Lung_Opacity/masks/Lung_Opacity-3298.png,True,False,299,299,L,71.98,899c2010a691141a55dab77acd68770a
Lung_Opacity-3299.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3299.png,Lung_Opacity/images/Lung_Opacity-3299.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3299.png,Lung_Opacity/masks/Lung_Opacity-3299.png,True,False,299,299,L,155.15,64dda48aadac8bceaca85b70f8d65d81
Lung_Opacity-33.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-33.png,Lung_Opacity/images/Lung_Opacity-33.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-33.png,Lung_Opacity/masks/Lung_Opacity-33.png,True,False,299,299,L,99.41,ca2472b567c9a20bd352dded3ec65b3d
Lung_Opacity-330.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-330.png,Lung_Opacity/images/Lung_Opacity-330.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-330.png,Lung_Opacity/masks/Lung_Opacity-330.png,True,False,299,299,L,121.8,fc7093fec79cb26bf61ed45d7d1b18f8
Lung_Opacity-3300.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3300.png,Lung_Opacity/images/Lung_Opacity-3300.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3300.png,Lung_Opacity/masks/Lung_Opacity-3300.png,True,False,299,299,L,127.42,918001c32060e0e29412b5cf0f4b8d9a
Lung_Opacity-3301.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3301.png,Lung_Opacity/images/Lung_Opacity-3301.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3301.png,Lung_Opacity/masks/Lung_Opacity-3301.png,True,False,299,299,L,125.56,400976a5c9298fcbc81b7235635f7194
Lung_Opacity-3302.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3302.png,Lung_Opacity/images/Lung_Opacity-3302.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3302.png,Lung_Opacity/masks/Lung_Opacity-3302.png,True,False,299,299,L,106.95,3e87547b741b41b6e47601e7c30f065f
Lung_Opacity-3303.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3303.png,Lung_Opacity/images/Lung_Opacity-3303.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3303.png,Lung_Opacity/masks/Lung_Opacity-3303.png,True,False,299,299,L,114.18,f55f0c95070b27e80426af88684f4d56
Lung_Opacity-3304.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3304.png,Lung_Opacity/images/Lung_Opacity-3304.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3304.png,Lung_Opacity/masks/Lung_Opacity-3304.png,True,False,299,299,L,116.86,75c5a5544f6eba0ac929eb8bd6206d15
Lung_Opacity-3305.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3305.png,Lung_Opacity/images/Lung_Opacity-3305.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3305.png,Lung_Opacity/masks/Lung_Opacity-3305.png,True,False,299,299,L,58.96,25b4b4d56109358d6c253dbd6d8e3027
Lung_Opacity-3306.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3306.png,Lung_Opacity/images/Lung_Opacity-3306.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3306.png,Lung_Opacity/masks/Lung_Opacity-3306.png,True,False,299,299,L,142.42,1b8abfd034cd9b059439a67a02e4d79a
Lung_Opacity-3307.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3307.png,Lung_Opacity/images/Lung_Opacity-3307.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3307.png,Lung_Opacity/masks/Lung_Opacity-3307.png,True,False,299,299,L,180.13,2e1d8d3b6298d1aecbdca67886fcad91
Lung_Opacity-3308.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3308.png,Lung_Opacity/images/Lung_Opacity-3308.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3308.png,Lung_Opacity/masks/Lung_Opacity-3308.png,True,False,299,299,L,157.0,3bea55e68f8b0f653cfdfc3c55ec7239
Lung_Opacity-3309.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3309.png,Lung_Opacity/images/Lung_Opacity-3309.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3309.png,Lung_Opacity/masks/Lung_Opacity-3309.png,True,False,299,299,L,101.88,ed22bcafac899345c47640993b343371
Lung_Opacity-331.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-331.png,Lung_Opacity/images/Lung_Opacity-331.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-331.png,Lung_Opacity/masks/Lung_Opacity-331.png,True,False,299,299,L,101.94,85dacd8e4d360859d84ca1174113ea11
Lung_Opacity-3310.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3310.png,Lung_Opacity/images/Lung_Opacity-3310.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3310.png,Lung_Opacity/masks/Lung_Opacity-3310.png,True,False,299,299,L,114.18,88d6293de2815e6f3de036da90a91498
Lung_Opacity-3311.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3311.png,Lung_Opacity/images/Lung_Opacity-3311.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3311.png,Lung_Opacity/masks/Lung_Opacity-3311.png,True,False,299,299,L,92.85,0fac220307a4169209a9647b2395c6ca
Lung_Opacity-3312.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3312.png,Lung_Opacity/images/Lung_Opacity-3312.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3312.png,Lung_Opacity/masks/Lung_Opacity-3312.png,True,False,299,299,L,176.27,7bb4dab2baaacba4849dc29e25fa0a24
Lung_Opacity-3313.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3313.png,Lung_Opacity/images/Lung_Opacity-3313.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3313.png,Lung_Opacity/masks/Lung_Opacity-3313.png,True,False,299,299,L,123.25,833cb642bc97f12e3739b4b1fad358f3
Lung_Opacity-3314.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3314.png,Lung_Opacity/images/Lung_Opacity-3314.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3314.png,Lung_Opacity/masks/Lung_Opacity-3314.png,True,False,299,299,L,127.86,b70ab455d27fdfb285e8c9ee4664047a
Lung_Opacity-3315.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3315.png,Lung_Opacity/images/Lung_Opacity-3315.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3315.png,Lung_Opacity/masks/Lung_Opacity-3315.png,True,False,299,299,L,177.22,18e82eccdc152c24c68ad88e9dcfe3ef
Lung_Opacity-3316.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3316.png,Lung_Opacity/images/Lung_Opacity-3316.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3316.png,Lung_Opacity/masks/Lung_Opacity-3316.png,True,False,299,299,L,105.38,b160df9ea3b9d353af98243a5417e678
Lung_Opacity-3317.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3317.png,Lung_Opacity/images/Lung_Opacity-3317.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3317.png,Lung_Opacity/masks/Lung_Opacity-3317.png,True,False,299,299,L,111.92,969e9717b1d8c20e085c80c92cb7228e
Lung_Opacity-3318.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3318.png,Lung_Opacity/images/Lung_Opacity-3318.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3318.png,Lung_Opacity/masks/Lung_Opacity-3318.png,True,False,299,299,L,100.12,cc8b0c164c1b82f363f1d8fbdac188c4
Lung_Opacity-3319.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3319.png,Lung_Opacity/images/Lung_Opacity-3319.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3319.png,Lung_Opacity/masks/Lung_Opacity-3319.png,True,False,299,299,L,100.93,c9fba77d7e0e9059bf68f73d2824a081
Lung_Opacity-332.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-332.png,Lung_Opacity/images/Lung_Opacity-332.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-332.png,Lung_Opacity/masks/Lung_Opacity-332.png,True,False,299,299,L,140.13,0c5d46accdc0dedfb179898ef5a8d506
Lung_Opacity-3320.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3320.png,Lung_Opacity/images/Lung_Opacity-3320.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3320.png,Lung_Opacity/masks/Lung_Opacity-3320.png,True,False,299,299,L,140.34,00f566e64046f0cc1576f382358066f6
Lung_Opacity-3321.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3321.png,Lung_Opacity/images/Lung_Opacity-3321.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3321.png,Lung_Opacity/masks/Lung_Opacity-3321.png,True,False,299,299,L,92.1,233dc28b3bc34ba5cca8f21c2da85506
Lung_Opacity-3322.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3322.png,Lung_Opacity/images/Lung_Opacity-3322.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3322.png,Lung_Opacity/masks/Lung_Opacity-3322.png,True,False,299,299,L,111.95,ec0ad48e89dac303f9c55ebeb67582f5
Lung_Opacity-3323.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3323.png,Lung_Opacity/images/Lung_Opacity-3323.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3323.png,Lung_Opacity/masks/Lung_Opacity-3323.png,True,False,299,299,L,113.95,da24d5570c5aa7790a948de1fe882287
Lung_Opacity-3324.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3324.png,Lung_Opacity/images/Lung_Opacity-3324.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3324.png,Lung_Opacity/masks/Lung_Opacity-3324.png,True,False,299,299,L,112.54,6a42c5aa3cfdfcf3a0a2117625d7b395
Lung_Opacity-3325.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3325.png,Lung_Opacity/images/Lung_Opacity-3325.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3325.png,Lung_Opacity/masks/Lung_Opacity-3325.png,True,False,299,299,L,104.52,40049e2dfaa7594af2fde92d1186fe2c
Lung_Opacity-3326.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3326.png,Lung_Opacity/images/Lung_Opacity-3326.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3326.png,Lung_Opacity/masks/Lung_Opacity-3326.png,True,False,299,299,L,141.57,bb8b0bc6d4f0eff952f75a38ad862f25
Lung_Opacity-3327.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3327.png,Lung_Opacity/images/Lung_Opacity-3327.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3327.png,Lung_Opacity/masks/Lung_Opacity-3327.png,True,False,299,299,L,116.64,ebfd7b0be68f4540dce3138c2ccc798a
Lung_Opacity-3328.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3328.png,Lung_Opacity/images/Lung_Opacity-3328.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3328.png,Lung_Opacity/masks/Lung_Opacity-3328.png,True,False,299,299,L,128.16,76361bea7ea07ab6e447c8960716fba9
Lung_Opacity-3329.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3329.png,Lung_Opacity/images/Lung_Opacity-3329.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3329.png,Lung_Opacity/masks/Lung_Opacity-3329.png,True,False,299,299,L,109.82,82ac57c5548a6ee25409a9ae1ee8196e
Lung_Opacity-333.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-333.png,Lung_Opacity/images/Lung_Opacity-333.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-333.png,Lung_Opacity/masks/Lung_Opacity-333.png,True,False,299,299,L,121.32,1e890df38cf2c9bb9e5d3f59d087e16e
Lung_Opacity-3330.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3330.png,Lung_Opacity/images/Lung_Opacity-3330.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3330.png,Lung_Opacity/masks/Lung_Opacity-3330.png,True,False,299,299,L,99.99,bf6ab5101d9397781da163566ff5a5d7
Lung_Opacity-3331.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3331.png,Lung_Opacity/images/Lung_Opacity-3331.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3331.png,Lung_Opacity/masks/Lung_Opacity-3331.png,True,False,299,299,L,101.66,6adbf0802c06be3f750c8bdd9a523678
Lung_Opacity-3332.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3332.png,Lung_Opacity/images/Lung_Opacity-3332.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3332.png,Lung_Opacity/masks/Lung_Opacity-3332.png,True,False,299,299,L,99.97,b5d159ca78b32340f508ba9b353f7328
Lung_Opacity-3333.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3333.png,Lung_Opacity/images/Lung_Opacity-3333.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3333.png,Lung_Opacity/masks/Lung_Opacity-3333.png,True,False,299,299,L,101.66,ddb53bd22311b610961b894d3411520a
Lung_Opacity-3334.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3334.png,Lung_Opacity/images/Lung_Opacity-3334.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3334.png,Lung_Opacity/masks/Lung_Opacity-3334.png,True,False,299,299,L,172.83,7aa8e91492d46bfa1eccff40e4699505
Lung_Opacity-3335.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3335.png,Lung_Opacity/images/Lung_Opacity-3335.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3335.png,Lung_Opacity/masks/Lung_Opacity-3335.png,True,False,299,299,L,99.88,c15468b562f468d0c8099d36f1a97892
Lung_Opacity-3336.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3336.png,Lung_Opacity/images/Lung_Opacity-3336.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3336.png,Lung_Opacity/masks/Lung_Opacity-3336.png,True,False,299,299,L,70.57,8671fb277b92726a61e60c652f33e366
Lung_Opacity-3337.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3337.png,Lung_Opacity/images/Lung_Opacity-3337.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3337.png,Lung_Opacity/masks/Lung_Opacity-3337.png,True,False,299,299,L,107.45,cbaaf79b60883f2b51f991fd63d431e4
Lung_Opacity-3338.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3338.png,Lung_Opacity/images/Lung_Opacity-3338.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3338.png,Lung_Opacity/masks/Lung_Opacity-3338.png,True,False,299,299,L,99.49,12bea7bd25bd7447093448ef94a55324
Lung_Opacity-3339.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3339.png,Lung_Opacity/images/Lung_Opacity-3339.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3339.png,Lung_Opacity/masks/Lung_Opacity-3339.png,True,False,299,299,L,119.25,57ccc854090658211914a79dfe39e707
Lung_Opacity-334.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-334.png,Lung_Opacity/images/Lung_Opacity-334.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-334.png,Lung_Opacity/masks/Lung_Opacity-334.png,True,False,299,299,L,170.25,1f870ae88999ffe36655ef1c21da2b3c
Lung_Opacity-3340.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3340.png,Lung_Opacity/images/Lung_Opacity-3340.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3340.png,Lung_Opacity/masks/Lung_Opacity-3340.png,True,False,299,299,L,127.87,2da27264a68ea95ed80579d13f2857e5
Lung_Opacity-3341.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3341.png,Lung_Opacity/images/Lung_Opacity-3341.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3341.png,Lung_Opacity/masks/Lung_Opacity-3341.png,True,False,299,299,L,119.4,a4b7024ce3fd6967a349885943218361
Lung_Opacity-3342.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3342.png,Lung_Opacity/images/Lung_Opacity-3342.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3342.png,Lung_Opacity/masks/Lung_Opacity-3342.png,True,False,299,299,L,138.64,0ed227e2b925af47248d2399f444e461
Lung_Opacity-3343.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3343.png,Lung_Opacity/images/Lung_Opacity-3343.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3343.png,Lung_Opacity/masks/Lung_Opacity-3343.png,True,False,299,299,L,128.26,57928ee18b0d649fc75f8a9a1446e42f
Lung_Opacity-3344.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3344.png,Lung_Opacity/images/Lung_Opacity-3344.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3344.png,Lung_Opacity/masks/Lung_Opacity-3344.png,True,False,299,299,L,134.59,7d0403555c75495ed627fa356899ba58
Lung_Opacity-3345.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3345.png,Lung_Opacity/images/Lung_Opacity-3345.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3345.png,Lung_Opacity/masks/Lung_Opacity-3345.png,True,False,299,299,L,110.43,57c702e0ff1352d25d4f6f0d15b0979f
Lung_Opacity-3346.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3346.png,Lung_Opacity/images/Lung_Opacity-3346.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3346.png,Lung_Opacity/masks/Lung_Opacity-3346.png,True,False,299,299,L,101.26,640e1de2875372e66bec12e8bf0d0bbf
Lung_Opacity-3347.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3347.png,Lung_Opacity/images/Lung_Opacity-3347.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3347.png,Lung_Opacity/masks/Lung_Opacity-3347.png,True,False,299,299,L,84.58,9e2b0de7dfa3b888166e77bef2badf57
Lung_Opacity-3348.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3348.png,Lung_Opacity/images/Lung_Opacity-3348.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3348.png,Lung_Opacity/masks/Lung_Opacity-3348.png,True,False,299,299,L,174.23,bd389b761613cbcb9d91a491267a0f9d
Lung_Opacity-3349.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3349.png,Lung_Opacity/images/Lung_Opacity-3349.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3349.png,Lung_Opacity/masks/Lung_Opacity-3349.png,True,False,299,299,L,115.18,99ee56b86c7958d56867e733a61d57b0
Lung_Opacity-335.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-335.png,Lung_Opacity/images/Lung_Opacity-335.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-335.png,Lung_Opacity/masks/Lung_Opacity-335.png,True,False,299,299,L,113.85,c049f7aaee2176a1a03b7408d708a264
Lung_Opacity-3350.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3350.png,Lung_Opacity/images/Lung_Opacity-3350.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3350.png,Lung_Opacity/masks/Lung_Opacity-3350.png,True,False,299,299,L,113.34,360d8557264db61503f0d98dd595e54a
Lung_Opacity-3351.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3351.png,Lung_Opacity/images/Lung_Opacity-3351.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3351.png,Lung_Opacity/masks/Lung_Opacity-3351.png,True,False,299,299,L,152.39,ee318d5f5f309a97b3ee0551861d51a8
Lung_Opacity-3352.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3352.png,Lung_Opacity/images/Lung_Opacity-3352.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3352.png,Lung_Opacity/masks/Lung_Opacity-3352.png,True,False,299,299,L,109.7,36d88132830eebafc1d895ad4aed6ba0
Lung_Opacity-3353.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3353.png,Lung_Opacity/images/Lung_Opacity-3353.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3353.png,Lung_Opacity/masks/Lung_Opacity-3353.png,True,False,299,299,L,125.23,69567af9f43f3ba45c2892c36c919f31
Lung_Opacity-3354.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3354.png,Lung_Opacity/images/Lung_Opacity-3354.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3354.png,Lung_Opacity/masks/Lung_Opacity-3354.png,True,False,299,299,L,106.24,d82f2bd1251815d87ba0656fda193449
Lung_Opacity-3355.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3355.png,Lung_Opacity/images/Lung_Opacity-3355.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3355.png,Lung_Opacity/masks/Lung_Opacity-3355.png,True,False,299,299,L,106.26,9c8833b2ecf6790d7b0ef8c7b4ec2daa
Lung_Opacity-3356.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3356.png,Lung_Opacity/images/Lung_Opacity-3356.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3356.png,Lung_Opacity/masks/Lung_Opacity-3356.png,True,False,299,299,L,124.4,62a4332361ee70e0178da1b7a48e9288
Lung_Opacity-3357.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3357.png,Lung_Opacity/images/Lung_Opacity-3357.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3357.png,Lung_Opacity/masks/Lung_Opacity-3357.png,True,False,299,299,L,102.84,917b9d91fc47f52c1314185c421e6617
Lung_Opacity-3358.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3358.png,Lung_Opacity/images/Lung_Opacity-3358.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3358.png,Lung_Opacity/masks/Lung_Opacity-3358.png,True,False,299,299,L,154.96,28fba30adefe9b51ce91cfa4e25ef8a9
Lung_Opacity-3359.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3359.png,Lung_Opacity/images/Lung_Opacity-3359.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3359.png,Lung_Opacity/masks/Lung_Opacity-3359.png,True,False,299,299,L,140.58,56c7bd885220c090d86c3701b18e95f7
Lung_Opacity-336.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-336.png,Lung_Opacity/images/Lung_Opacity-336.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-336.png,Lung_Opacity/masks/Lung_Opacity-336.png,True,False,299,299,L,156.48,45a0974d21acb6081397b047106ff688
Lung_Opacity-3360.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3360.png,Lung_Opacity/images/Lung_Opacity-3360.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3360.png,Lung_Opacity/masks/Lung_Opacity-3360.png,True,False,299,299,L,173.71,fed9402067452a38738f56c4afe46c07
Lung_Opacity-3361.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3361.png,Lung_Opacity/images/Lung_Opacity-3361.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3361.png,Lung_Opacity/masks/Lung_Opacity-3361.png,True,False,299,299,L,108.59,13c8b1b9901324e9a4309d695adda98a
Lung_Opacity-3362.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3362.png,Lung_Opacity/images/Lung_Opacity-3362.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3362.png,Lung_Opacity/masks/Lung_Opacity-3362.png,True,False,299,299,L,153.18,07303a29e5d2a088809a986dd4d2f297
Lung_Opacity-3363.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3363.png,Lung_Opacity/images/Lung_Opacity-3363.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3363.png,Lung_Opacity/masks/Lung_Opacity-3363.png,True,False,299,299,L,167.63,cfd2d2cf309228cddb40a093811dad9e
Lung_Opacity-3364.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3364.png,Lung_Opacity/images/Lung_Opacity-3364.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3364.png,Lung_Opacity/masks/Lung_Opacity-3364.png,True,False,299,299,L,90.52,1f65d5024336a4eb37c0659211160198
Lung_Opacity-3365.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3365.png,Lung_Opacity/images/Lung_Opacity-3365.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3365.png,Lung_Opacity/masks/Lung_Opacity-3365.png,True,False,299,299,L,119.58,e317dd33a84b80885269b7055b07a9b2
Lung_Opacity-3366.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3366.png,Lung_Opacity/images/Lung_Opacity-3366.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3366.png,Lung_Opacity/masks/Lung_Opacity-3366.png,True,False,299,299,L,113.18,846760148a5b78103c856216e551498f
Lung_Opacity-3367.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3367.png,Lung_Opacity/images/Lung_Opacity-3367.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3367.png,Lung_Opacity/masks/Lung_Opacity-3367.png,True,False,299,299,L,112.12,797bba2a6a637dd90eb98aa3ae1cfc8d
Lung_Opacity-3368.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3368.png,Lung_Opacity/images/Lung_Opacity-3368.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3368.png,Lung_Opacity/masks/Lung_Opacity-3368.png,True,False,299,299,L,127.14,8b825c5ce2704f076c6f0ae00cc5b4d3
Lung_Opacity-3369.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3369.png,Lung_Opacity/images/Lung_Opacity-3369.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3369.png,Lung_Opacity/masks/Lung_Opacity-3369.png,True,False,299,299,L,129.28,f6ce34a30d629e913c820b74496a0c49
Lung_Opacity-337.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-337.png,Lung_Opacity/images/Lung_Opacity-337.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-337.png,Lung_Opacity/masks/Lung_Opacity-337.png,True,False,299,299,L,110.77,f74172b3a7cab1b0c0dceda9d6a78b83
Lung_Opacity-3370.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3370.png,Lung_Opacity/images/Lung_Opacity-3370.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3370.png,Lung_Opacity/masks/Lung_Opacity-3370.png,True,False,299,299,L,114.94,8b7860c6354ddc824590a8bce990ef7c
Lung_Opacity-3371.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3371.png,Lung_Opacity/images/Lung_Opacity-3371.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3371.png,Lung_Opacity/masks/Lung_Opacity-3371.png,True,False,299,299,L,163.8,0c8295c14527dd2849242beeabc902dc
Lung_Opacity-3372.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3372.png,Lung_Opacity/images/Lung_Opacity-3372.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3372.png,Lung_Opacity/masks/Lung_Opacity-3372.png,True,False,299,299,L,132.94,b75c2fd4fb761eb7792cffe3730a006f
Lung_Opacity-3373.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3373.png,Lung_Opacity/images/Lung_Opacity-3373.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3373.png,Lung_Opacity/masks/Lung_Opacity-3373.png,True,False,299,299,L,83.76,a19724b95aea7e3637c374a1b1974951
Lung_Opacity-3374.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3374.png,Lung_Opacity/images/Lung_Opacity-3374.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3374.png,Lung_Opacity/masks/Lung_Opacity-3374.png,True,False,299,299,L,106.89,1de3988fcc7dfff5efebb4779aa1f0c5
Lung_Opacity-3375.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3375.png,Lung_Opacity/images/Lung_Opacity-3375.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3375.png,Lung_Opacity/masks/Lung_Opacity-3375.png,True,False,299,299,L,109.63,41452f9f8876f474978590ca9d308697
Lung_Opacity-3376.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3376.png,Lung_Opacity/images/Lung_Opacity-3376.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3376.png,Lung_Opacity/masks/Lung_Opacity-3376.png,True,False,299,299,L,154.94,35fd9fc92e43c67d6afd467a0098acbd
Lung_Opacity-3377.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3377.png,Lung_Opacity/images/Lung_Opacity-3377.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3377.png,Lung_Opacity/masks/Lung_Opacity-3377.png,True,False,299,299,L,143.43,06ee2bc1ae5aefb492156ed28032fde4
Lung_Opacity-3378.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3378.png,Lung_Opacity/images/Lung_Opacity-3378.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3378.png,Lung_Opacity/masks/Lung_Opacity-3378.png,True,False,299,299,L,117.79,026a3b38248d60e72463dad8630fdc16
Lung_Opacity-3379.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3379.png,Lung_Opacity/images/Lung_Opacity-3379.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3379.png,Lung_Opacity/masks/Lung_Opacity-3379.png,True,False,299,299,L,145.49,b8216e0667b2101c6df4b2331e55bb25
Lung_Opacity-338.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-338.png,Lung_Opacity/images/Lung_Opacity-338.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-338.png,Lung_Opacity/masks/Lung_Opacity-338.png,True,False,299,299,L,160.38,a7f6a2e0833bdba79d48f41e05d37617
Lung_Opacity-3380.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3380.png,Lung_Opacity/images/Lung_Opacity-3380.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3380.png,Lung_Opacity/masks/Lung_Opacity-3380.png,True,False,299,299,L,121.91,9fabb3269d30a38bebb2d34a69741926
Lung_Opacity-3381.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3381.png,Lung_Opacity/images/Lung_Opacity-3381.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3381.png,Lung_Opacity/masks/Lung_Opacity-3381.png,True,False,299,299,L,51.73,27877a6f261c49013b6701d7d0f66f92
Lung_Opacity-3382.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3382.png,Lung_Opacity/images/Lung_Opacity-3382.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3382.png,Lung_Opacity/masks/Lung_Opacity-3382.png,True,False,299,299,L,120.56,6a63d0d67c2233f000396f725b5b8ae4
Lung_Opacity-3383.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3383.png,Lung_Opacity/images/Lung_Opacity-3383.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3383.png,Lung_Opacity/masks/Lung_Opacity-3383.png,True,False,299,299,L,116.43,2e05a1f0c6671c9242f4452b7fde3bdd
Lung_Opacity-3384.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3384.png,Lung_Opacity/images/Lung_Opacity-3384.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3384.png,Lung_Opacity/masks/Lung_Opacity-3384.png,True,False,299,299,L,125.42,7d6001d8b5629c6bc342c8014f21d223
Lung_Opacity-3385.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3385.png,Lung_Opacity/images/Lung_Opacity-3385.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3385.png,Lung_Opacity/masks/Lung_Opacity-3385.png,True,False,299,299,L,124.2,a5c7f199be5340cd70c13c01852a9960
Lung_Opacity-3386.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3386.png,Lung_Opacity/images/Lung_Opacity-3386.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3386.png,Lung_Opacity/masks/Lung_Opacity-3386.png,True,False,299,299,L,156.35,882f60deaf06305e59b3a87e0f538301
Lung_Opacity-3387.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3387.png,Lung_Opacity/images/Lung_Opacity-3387.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3387.png,Lung_Opacity/masks/Lung_Opacity-3387.png,True,False,299,299,L,101.46,e9ad013eff9b148e4f00efea0bc37a81
Lung_Opacity-3388.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3388.png,Lung_Opacity/images/Lung_Opacity-3388.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3388.png,Lung_Opacity/masks/Lung_Opacity-3388.png,True,False,299,299,L,169.36,7695fb6feafe8624c2a17e3eaf8c08e2
Lung_Opacity-3389.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3389.png,Lung_Opacity/images/Lung_Opacity-3389.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3389.png,Lung_Opacity/masks/Lung_Opacity-3389.png,True,False,299,299,L,74.95,8561b5c653b6f0fb75ef5cc096c62e4b
Lung_Opacity-339.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-339.png,Lung_Opacity/images/Lung_Opacity-339.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-339.png,Lung_Opacity/masks/Lung_Opacity-339.png,True,False,299,299,L,127.16,a6c894fc132d112dccd225fa30cd2d5f
Lung_Opacity-3390.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3390.png,Lung_Opacity/images/Lung_Opacity-3390.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3390.png,Lung_Opacity/masks/Lung_Opacity-3390.png,True,False,299,299,L,105.95,dc1cae7c9ba0108d2b74ec7e7d648c85
Lung_Opacity-3391.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3391.png,Lung_Opacity/images/Lung_Opacity-3391.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3391.png,Lung_Opacity/masks/Lung_Opacity-3391.png,True,False,299,299,L,171.9,cbbccb5ef18727b15d255228f394e41a
Lung_Opacity-3392.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3392.png,Lung_Opacity/images/Lung_Opacity-3392.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3392.png,Lung_Opacity/masks/Lung_Opacity-3392.png,True,False,299,299,L,125.27,0c699a1ee48309e65e58bca57a3e9259
Lung_Opacity-3393.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3393.png,Lung_Opacity/images/Lung_Opacity-3393.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3393.png,Lung_Opacity/masks/Lung_Opacity-3393.png,True,False,299,299,L,136.68,1cc9b9c23e1e096237310b0bf6b7b5bf
Lung_Opacity-3394.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3394.png,Lung_Opacity/images/Lung_Opacity-3394.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3394.png,Lung_Opacity/masks/Lung_Opacity-3394.png,True,False,299,299,L,140.23,88369120ef0a42594570863a890d4ba0
Lung_Opacity-3395.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3395.png,Lung_Opacity/images/Lung_Opacity-3395.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3395.png,Lung_Opacity/masks/Lung_Opacity-3395.png,True,False,299,299,L,124.04,8a721bffb2a46301b805706c58d0b6a9
Lung_Opacity-3396.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3396.png,Lung_Opacity/images/Lung_Opacity-3396.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3396.png,Lung_Opacity/masks/Lung_Opacity-3396.png,True,False,299,299,L,118.2,6f0f65dbc0bf550bf47b33a7c44615e6
Lung_Opacity-3397.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3397.png,Lung_Opacity/images/Lung_Opacity-3397.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3397.png,Lung_Opacity/masks/Lung_Opacity-3397.png,True,False,299,299,L,127.96,c43ad37e46231f3cbf3ac18e31d6c2d9
Lung_Opacity-3398.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3398.png,Lung_Opacity/images/Lung_Opacity-3398.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3398.png,Lung_Opacity/masks/Lung_Opacity-3398.png,True,False,299,299,L,106.57,7237351a44398af12896907061c09e97
Lung_Opacity-3399.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3399.png,Lung_Opacity/images/Lung_Opacity-3399.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3399.png,Lung_Opacity/masks/Lung_Opacity-3399.png,True,False,299,299,L,134.45,dc3605e433ac1581c9e96d6bd484875e
Lung_Opacity-34.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-34.png,Lung_Opacity/images/Lung_Opacity-34.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-34.png,Lung_Opacity/masks/Lung_Opacity-34.png,True,False,299,299,L,118.59,09c05b3089808fcbc5e5beedb6e0081e
Lung_Opacity-340.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-340.png,Lung_Opacity/images/Lung_Opacity-340.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-340.png,Lung_Opacity/masks/Lung_Opacity-340.png,True,False,299,299,L,130.41,68734dfc7146cfc3035893b9e07dd807
Lung_Opacity-3400.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3400.png,Lung_Opacity/images/Lung_Opacity-3400.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3400.png,Lung_Opacity/masks/Lung_Opacity-3400.png,True,False,299,299,L,104.73,10059f2a12d19fec0d0c1c837982a5f3
Lung_Opacity-3401.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3401.png,Lung_Opacity/images/Lung_Opacity-3401.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3401.png,Lung_Opacity/masks/Lung_Opacity-3401.png,True,False,299,299,L,115.94,76a3ae08bb4e198f0a9ff7fe4091be05
Lung_Opacity-3402.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3402.png,Lung_Opacity/images/Lung_Opacity-3402.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3402.png,Lung_Opacity/masks/Lung_Opacity-3402.png,True,False,299,299,L,117.4,42b027abe9f54269d2f38ca8b1533cb8
Lung_Opacity-3403.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3403.png,Lung_Opacity/images/Lung_Opacity-3403.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3403.png,Lung_Opacity/masks/Lung_Opacity-3403.png,True,False,299,299,L,112.87,a2baec7ebb2609c399338494b2acdf4a
Lung_Opacity-3404.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3404.png,Lung_Opacity/images/Lung_Opacity-3404.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3404.png,Lung_Opacity/masks/Lung_Opacity-3404.png,True,False,299,299,L,124.8,6f4a5b33e7c121c57cacafe43a5c5ce4
Lung_Opacity-3405.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3405.png,Lung_Opacity/images/Lung_Opacity-3405.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3405.png,Lung_Opacity/masks/Lung_Opacity-3405.png,True,False,299,299,L,167.09,13fca877b23919fd48e3b58fd462a9f1
Lung_Opacity-3406.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3406.png,Lung_Opacity/images/Lung_Opacity-3406.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3406.png,Lung_Opacity/masks/Lung_Opacity-3406.png,True,False,299,299,L,114.51,524258834a508492a7be6e78382adb73
Lung_Opacity-3407.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3407.png,Lung_Opacity/images/Lung_Opacity-3407.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3407.png,Lung_Opacity/masks/Lung_Opacity-3407.png,True,False,299,299,L,149.08,e24f8ed35179ec96ceea78be717bb065
Lung_Opacity-3408.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3408.png,Lung_Opacity/images/Lung_Opacity-3408.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3408.png,Lung_Opacity/masks/Lung_Opacity-3408.png,True,False,299,299,L,113.74,87782539bb10b42875ff5895ac7b43b2
Lung_Opacity-3409.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3409.png,Lung_Opacity/images/Lung_Opacity-3409.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3409.png,Lung_Opacity/masks/Lung_Opacity-3409.png,True,False,299,299,L,101.7,a5affd4a11f7750d4d55280ba3d47af0
Lung_Opacity-341.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-341.png,Lung_Opacity/images/Lung_Opacity-341.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-341.png,Lung_Opacity/masks/Lung_Opacity-341.png,True,False,299,299,L,95.38,6bb18568c39033eb08615931de57ba54
Lung_Opacity-3410.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3410.png,Lung_Opacity/images/Lung_Opacity-3410.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3410.png,Lung_Opacity/masks/Lung_Opacity-3410.png,True,False,299,299,L,123.41,173e20fe8ec44ee03d26c4e1be1f91f6
Lung_Opacity-3411.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3411.png,Lung_Opacity/images/Lung_Opacity-3411.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3411.png,Lung_Opacity/masks/Lung_Opacity-3411.png,True,False,299,299,L,125.41,4feaa28da35e9498086234483c7375e6
Lung_Opacity-3412.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3412.png,Lung_Opacity/images/Lung_Opacity-3412.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3412.png,Lung_Opacity/masks/Lung_Opacity-3412.png,True,False,299,299,L,101.86,c807c6eb0d4604faaffc9ffa8d431e74
Lung_Opacity-3413.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3413.png,Lung_Opacity/images/Lung_Opacity-3413.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3413.png,Lung_Opacity/masks/Lung_Opacity-3413.png,True,False,299,299,L,120.39,e21cfdceebe4f95162570a83c5a8b6a4
Lung_Opacity-3414.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3414.png,Lung_Opacity/images/Lung_Opacity-3414.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3414.png,Lung_Opacity/masks/Lung_Opacity-3414.png,True,False,299,299,L,115.2,0bc313d000a63215585c1d70c468855b
Lung_Opacity-3415.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3415.png,Lung_Opacity/images/Lung_Opacity-3415.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3415.png,Lung_Opacity/masks/Lung_Opacity-3415.png,True,False,299,299,L,129.06,b8d72cf7083cbd3dd79d86287931b319
Lung_Opacity-3416.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3416.png,Lung_Opacity/images/Lung_Opacity-3416.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3416.png,Lung_Opacity/masks/Lung_Opacity-3416.png,True,False,299,299,L,153.06,12f5e2d6b6c6b43dc84554e0b766e7a2
Lung_Opacity-3417.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3417.png,Lung_Opacity/images/Lung_Opacity-3417.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3417.png,Lung_Opacity/masks/Lung_Opacity-3417.png,True,False,299,299,L,127.87,6f568d20a2015da5c1c86a6d40b092b0
Lung_Opacity-3418.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3418.png,Lung_Opacity/images/Lung_Opacity-3418.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3418.png,Lung_Opacity/masks/Lung_Opacity-3418.png,True,False,299,299,L,150.83,ca25a07e7ed6039481b42d77c0270da8
Lung_Opacity-3419.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3419.png,Lung_Opacity/images/Lung_Opacity-3419.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3419.png,Lung_Opacity/masks/Lung_Opacity-3419.png,True,False,299,299,L,140.03,ee8672ef929db64d665644b965661e9c
Lung_Opacity-342.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-342.png,Lung_Opacity/images/Lung_Opacity-342.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-342.png,Lung_Opacity/masks/Lung_Opacity-342.png,True,False,299,299,L,114.45,5119c8c3338f4910443e10834f24f6cd
Lung_Opacity-3420.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3420.png,Lung_Opacity/images/Lung_Opacity-3420.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3420.png,Lung_Opacity/masks/Lung_Opacity-3420.png,True,False,299,299,L,141.56,b88e1eed36365ca45046dbc8b8a8673c
Lung_Opacity-3421.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3421.png,Lung_Opacity/images/Lung_Opacity-3421.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3421.png,Lung_Opacity/masks/Lung_Opacity-3421.png,True,False,299,299,L,95.17,490f60e6425f867ceb1cc7059d59a5d7
Lung_Opacity-3422.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3422.png,Lung_Opacity/images/Lung_Opacity-3422.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3422.png,Lung_Opacity/masks/Lung_Opacity-3422.png,True,False,299,299,L,125.44,ec33199938763a626b362da3f4abe20f
Lung_Opacity-3423.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3423.png,Lung_Opacity/images/Lung_Opacity-3423.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3423.png,Lung_Opacity/masks/Lung_Opacity-3423.png,True,False,299,299,L,159.31,90961bdebfc9357bcecacfe174a980ad
Lung_Opacity-3424.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3424.png,Lung_Opacity/images/Lung_Opacity-3424.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3424.png,Lung_Opacity/masks/Lung_Opacity-3424.png,True,False,299,299,L,115.69,2f764bf7679725cb7773da3179317c71
Lung_Opacity-3425.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3425.png,Lung_Opacity/images/Lung_Opacity-3425.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3425.png,Lung_Opacity/masks/Lung_Opacity-3425.png,True,False,299,299,L,110.87,7d640a530746feea9eea08c7956ba8e0
Lung_Opacity-3426.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3426.png,Lung_Opacity/images/Lung_Opacity-3426.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3426.png,Lung_Opacity/masks/Lung_Opacity-3426.png,True,False,299,299,L,110.8,1893041b09375b52335a57a20c251fec
Lung_Opacity-3427.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3427.png,Lung_Opacity/images/Lung_Opacity-3427.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3427.png,Lung_Opacity/masks/Lung_Opacity-3427.png,True,False,299,299,L,165.56,76670c2a74737c22c88a2eb82524feb9
Lung_Opacity-3428.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3428.png,Lung_Opacity/images/Lung_Opacity-3428.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3428.png,Lung_Opacity/masks/Lung_Opacity-3428.png,True,False,299,299,L,173.03,ab8bfb68fae1448766e30cbca28360a1
Lung_Opacity-3429.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3429.png,Lung_Opacity/images/Lung_Opacity-3429.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3429.png,Lung_Opacity/masks/Lung_Opacity-3429.png,True,False,299,299,L,97.98,7d1e594ffc7c753d6d198498b7922414
Lung_Opacity-343.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-343.png,Lung_Opacity/images/Lung_Opacity-343.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-343.png,Lung_Opacity/masks/Lung_Opacity-343.png,True,False,299,299,L,134.65,7a7a4f72326eb59627626a71dfd5f02d
Lung_Opacity-3430.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3430.png,Lung_Opacity/images/Lung_Opacity-3430.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3430.png,Lung_Opacity/masks/Lung_Opacity-3430.png,True,False,299,299,L,159.65,62aa0a9b1ad5dea155d1d3a99605b765
Lung_Opacity-3431.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3431.png,Lung_Opacity/images/Lung_Opacity-3431.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3431.png,Lung_Opacity/masks/Lung_Opacity-3431.png,True,False,299,299,L,114.28,8c92ba28048eb5017e7b796b953eb0fd
Lung_Opacity-3432.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3432.png,Lung_Opacity/images/Lung_Opacity-3432.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3432.png,Lung_Opacity/masks/Lung_Opacity-3432.png,True,False,299,299,L,127.57,2ec27187951fec53c386c59c5039b0e0
Lung_Opacity-3433.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3433.png,Lung_Opacity/images/Lung_Opacity-3433.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3433.png,Lung_Opacity/masks/Lung_Opacity-3433.png,True,False,299,299,L,118.0,69464e21f9f32fa7d40a2d78dc59191a
Lung_Opacity-3434.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3434.png,Lung_Opacity/images/Lung_Opacity-3434.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3434.png,Lung_Opacity/masks/Lung_Opacity-3434.png,True,False,299,299,L,110.72,dad58836a6abdcf97d1ef7d1afbae36e
Lung_Opacity-3435.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3435.png,Lung_Opacity/images/Lung_Opacity-3435.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3435.png,Lung_Opacity/masks/Lung_Opacity-3435.png,True,False,299,299,L,123.91,5a5c86ff2e8d6dc52ddf414936efd432
Lung_Opacity-3436.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3436.png,Lung_Opacity/images/Lung_Opacity-3436.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3436.png,Lung_Opacity/masks/Lung_Opacity-3436.png,True,False,299,299,L,104.18,10c105f217152d9a3c99eb844fea60b5
Lung_Opacity-3437.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3437.png,Lung_Opacity/images/Lung_Opacity-3437.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3437.png,Lung_Opacity/masks/Lung_Opacity-3437.png,True,False,299,299,L,135.41,ef5a2f36ddc90c2b61567c07e8530cba
Lung_Opacity-3438.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3438.png,Lung_Opacity/images/Lung_Opacity-3438.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3438.png,Lung_Opacity/masks/Lung_Opacity-3438.png,True,False,299,299,L,109.18,8cc35dd094589d45db781fc4d5482b80
Lung_Opacity-3439.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3439.png,Lung_Opacity/images/Lung_Opacity-3439.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3439.png,Lung_Opacity/masks/Lung_Opacity-3439.png,True,False,299,299,L,87.53,92238bcd319d21340c0e14f14013bf3e
Lung_Opacity-344.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-344.png,Lung_Opacity/images/Lung_Opacity-344.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-344.png,Lung_Opacity/masks/Lung_Opacity-344.png,True,False,299,299,L,145.58,40bfcedab51dc39d9795f0831867badd
Lung_Opacity-3440.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3440.png,Lung_Opacity/images/Lung_Opacity-3440.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3440.png,Lung_Opacity/masks/Lung_Opacity-3440.png,True,False,299,299,L,92.16,d22a3ca1e31013b190685f644b83f0fd
Lung_Opacity-3441.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3441.png,Lung_Opacity/images/Lung_Opacity-3441.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3441.png,Lung_Opacity/masks/Lung_Opacity-3441.png,True,False,299,299,L,118.05,0ed83f737ba0b25a74ec382ff4417eaf
Lung_Opacity-3442.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3442.png,Lung_Opacity/images/Lung_Opacity-3442.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3442.png,Lung_Opacity/masks/Lung_Opacity-3442.png,True,False,299,299,L,89.23,de5f226c5ef2d7680c288dd43ddfdffa
Lung_Opacity-3443.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3443.png,Lung_Opacity/images/Lung_Opacity-3443.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3443.png,Lung_Opacity/masks/Lung_Opacity-3443.png,True,False,299,299,L,163.88,ab5190d5bf6e679a9a0a575580621e06
Lung_Opacity-3444.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3444.png,Lung_Opacity/images/Lung_Opacity-3444.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3444.png,Lung_Opacity/masks/Lung_Opacity-3444.png,True,False,299,299,L,116.45,e9d2c5adbcb2b1c5a99e4199f53a923c
Lung_Opacity-3445.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3445.png,Lung_Opacity/images/Lung_Opacity-3445.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3445.png,Lung_Opacity/masks/Lung_Opacity-3445.png,True,False,299,299,L,112.4,28d9c44705413660b3c6b0ae1d8d7a03
Lung_Opacity-3446.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3446.png,Lung_Opacity/images/Lung_Opacity-3446.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3446.png,Lung_Opacity/masks/Lung_Opacity-3446.png,True,False,299,299,L,123.8,f460bd07d8d66136267550840d7c10ce
Lung_Opacity-3447.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3447.png,Lung_Opacity/images/Lung_Opacity-3447.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3447.png,Lung_Opacity/masks/Lung_Opacity-3447.png,True,False,299,299,L,110.15,de2d0bcaa087578691fecc310bd36774
Lung_Opacity-3448.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3448.png,Lung_Opacity/images/Lung_Opacity-3448.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3448.png,Lung_Opacity/masks/Lung_Opacity-3448.png,True,False,299,299,L,112.28,e7cdac7ad876c80540156b1ac371cadd
Lung_Opacity-3449.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3449.png,Lung_Opacity/images/Lung_Opacity-3449.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3449.png,Lung_Opacity/masks/Lung_Opacity-3449.png,True,False,299,299,L,104.4,1f05c3bd1b5809df3385c18696555771
Lung_Opacity-345.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-345.png,Lung_Opacity/images/Lung_Opacity-345.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-345.png,Lung_Opacity/masks/Lung_Opacity-345.png,True,False,299,299,L,158.35,18e2694001ff4774f349c0c2fd87a1b6
Lung_Opacity-3450.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3450.png,Lung_Opacity/images/Lung_Opacity-3450.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3450.png,Lung_Opacity/masks/Lung_Opacity-3450.png,True,False,299,299,L,120.5,716276530ba798c69f4642ca1e3dd1c4
Lung_Opacity-3451.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3451.png,Lung_Opacity/images/Lung_Opacity-3451.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3451.png,Lung_Opacity/masks/Lung_Opacity-3451.png,True,False,299,299,L,99.6,51d00872bc6f93f97967149a5f4ad365
Lung_Opacity-3452.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3452.png,Lung_Opacity/images/Lung_Opacity-3452.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3452.png,Lung_Opacity/masks/Lung_Opacity-3452.png,True,False,299,299,L,154.78,7b4a5c047532b25969cfda9c94c5df17
Lung_Opacity-3453.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3453.png,Lung_Opacity/images/Lung_Opacity-3453.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3453.png,Lung_Opacity/masks/Lung_Opacity-3453.png,True,False,299,299,L,174.85,ddc343d805f3d182bbbfff151ef25dfe
Lung_Opacity-3454.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3454.png,Lung_Opacity/images/Lung_Opacity-3454.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3454.png,Lung_Opacity/masks/Lung_Opacity-3454.png,True,False,299,299,L,115.53,7a02dcf866c595be4d140ccedcf69527
Lung_Opacity-3455.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3455.png,Lung_Opacity/images/Lung_Opacity-3455.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3455.png,Lung_Opacity/masks/Lung_Opacity-3455.png,True,False,299,299,L,115.59,00596bf04b019868b66387b5c0a947bb
Lung_Opacity-3456.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3456.png,Lung_Opacity/images/Lung_Opacity-3456.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3456.png,Lung_Opacity/masks/Lung_Opacity-3456.png,True,False,299,299,L,131.44,53bf28d706477bd8da3816bda45fc8a8
Lung_Opacity-3457.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3457.png,Lung_Opacity/images/Lung_Opacity-3457.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3457.png,Lung_Opacity/masks/Lung_Opacity-3457.png,True,False,299,299,L,118.24,37d1a4c84e24954fbc499ab5fe75c97b
Lung_Opacity-3458.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3458.png,Lung_Opacity/images/Lung_Opacity-3458.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3458.png,Lung_Opacity/masks/Lung_Opacity-3458.png,True,False,299,299,L,106.21,5ad92cc9ae76e7e673222a167732467d
Lung_Opacity-3459.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3459.png,Lung_Opacity/images/Lung_Opacity-3459.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3459.png,Lung_Opacity/masks/Lung_Opacity-3459.png,True,False,299,299,L,106.88,9c33686e18b86e9a50e7e9f0a16c8832
Lung_Opacity-346.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-346.png,Lung_Opacity/images/Lung_Opacity-346.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-346.png,Lung_Opacity/masks/Lung_Opacity-346.png,True,False,299,299,L,164.95,cede7be748b181afada46b6be41300b7
Lung_Opacity-3460.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3460.png,Lung_Opacity/images/Lung_Opacity-3460.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3460.png,Lung_Opacity/masks/Lung_Opacity-3460.png,True,False,299,299,L,66.91,9acf1264957f7992772f484161445f56
Lung_Opacity-3461.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3461.png,Lung_Opacity/images/Lung_Opacity-3461.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3461.png,Lung_Opacity/masks/Lung_Opacity-3461.png,True,False,299,299,L,120.44,d44579cdbacf56f40af0410847bf6330
Lung_Opacity-3462.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3462.png,Lung_Opacity/images/Lung_Opacity-3462.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3462.png,Lung_Opacity/masks/Lung_Opacity-3462.png,True,False,299,299,L,157.89,b7bd2262e7d72b6ad8b9dc6c60f3429f
Lung_Opacity-3463.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3463.png,Lung_Opacity/images/Lung_Opacity-3463.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3463.png,Lung_Opacity/masks/Lung_Opacity-3463.png,True,False,299,299,L,94.45,a7419f027ba5bc13b1175a8af73150c1
Lung_Opacity-3464.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3464.png,Lung_Opacity/images/Lung_Opacity-3464.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3464.png,Lung_Opacity/masks/Lung_Opacity-3464.png,True,False,299,299,L,107.43,8d441dcd3ba8d0f45f85edd5dde4668d
Lung_Opacity-3465.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3465.png,Lung_Opacity/images/Lung_Opacity-3465.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3465.png,Lung_Opacity/masks/Lung_Opacity-3465.png,True,False,299,299,L,112.35,caae512ba64e6afcb2d10f5ea7113a97
Lung_Opacity-3466.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3466.png,Lung_Opacity/images/Lung_Opacity-3466.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3466.png,Lung_Opacity/masks/Lung_Opacity-3466.png,True,False,299,299,L,113.81,f333edee95c7506ca5c7e00f7d295c59
Lung_Opacity-3467.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3467.png,Lung_Opacity/images/Lung_Opacity-3467.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3467.png,Lung_Opacity/masks/Lung_Opacity-3467.png,True,False,299,299,L,144.41,f97691c0342243701dd1c66c18c7bf2e
Lung_Opacity-3468.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3468.png,Lung_Opacity/images/Lung_Opacity-3468.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3468.png,Lung_Opacity/masks/Lung_Opacity-3468.png,True,False,299,299,L,140.31,66396f27b878c6597190de0cdb3c2c81
Lung_Opacity-3469.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3469.png,Lung_Opacity/images/Lung_Opacity-3469.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3469.png,Lung_Opacity/masks/Lung_Opacity-3469.png,True,False,299,299,L,102.43,981ea726d77afab90d9f85c4ab2de5e7
Lung_Opacity-347.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-347.png,Lung_Opacity/images/Lung_Opacity-347.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-347.png,Lung_Opacity/masks/Lung_Opacity-347.png,True,False,299,299,L,117.75,2eb345599b566c63e740c6549a794931
Lung_Opacity-3470.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3470.png,Lung_Opacity/images/Lung_Opacity-3470.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3470.png,Lung_Opacity/masks/Lung_Opacity-3470.png,True,False,299,299,L,106.7,c75c0d1da16c181f0cbdbbc69a661da5
Lung_Opacity-3471.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3471.png,Lung_Opacity/images/Lung_Opacity-3471.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3471.png,Lung_Opacity/masks/Lung_Opacity-3471.png,True,False,299,299,L,163.53,ee063671a9248a632135475d57a0708b
Lung_Opacity-3472.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3472.png,Lung_Opacity/images/Lung_Opacity-3472.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3472.png,Lung_Opacity/masks/Lung_Opacity-3472.png,True,False,299,299,L,114.33,a4c66b1018b53acf5167e6ab20f409e7
Lung_Opacity-3473.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3473.png,Lung_Opacity/images/Lung_Opacity-3473.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3473.png,Lung_Opacity/masks/Lung_Opacity-3473.png,True,False,299,299,L,81.02,1680ada53d0e7c169d1d923d3434c62d
Lung_Opacity-3474.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3474.png,Lung_Opacity/images/Lung_Opacity-3474.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3474.png,Lung_Opacity/masks/Lung_Opacity-3474.png,True,False,299,299,L,168.78,59090d558e47b7496cfc3906f5fa3345
Lung_Opacity-3475.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3475.png,Lung_Opacity/images/Lung_Opacity-3475.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3475.png,Lung_Opacity/masks/Lung_Opacity-3475.png,True,False,299,299,L,112.29,2172989a2e42b4a4291d05ad430c8f81
Lung_Opacity-3476.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3476.png,Lung_Opacity/images/Lung_Opacity-3476.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3476.png,Lung_Opacity/masks/Lung_Opacity-3476.png,True,False,299,299,L,96.54,3ab52d9c788405fb696b66190e8511a6
Lung_Opacity-3477.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3477.png,Lung_Opacity/images/Lung_Opacity-3477.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3477.png,Lung_Opacity/masks/Lung_Opacity-3477.png,True,False,299,299,L,120.64,86d1b9e4404bc096c61a27e9702b0b62
Lung_Opacity-3478.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3478.png,Lung_Opacity/images/Lung_Opacity-3478.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3478.png,Lung_Opacity/masks/Lung_Opacity-3478.png,True,False,299,299,L,161.43,8a2c10006a9e88732c8f846ab3be4b50
Lung_Opacity-3479.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3479.png,Lung_Opacity/images/Lung_Opacity-3479.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3479.png,Lung_Opacity/masks/Lung_Opacity-3479.png,True,False,299,299,L,158.02,d2688692dd5e0595dd93999111d22140
Lung_Opacity-348.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-348.png,Lung_Opacity/images/Lung_Opacity-348.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-348.png,Lung_Opacity/masks/Lung_Opacity-348.png,True,False,299,299,L,124.72,2988c34b587b92a339eb42acd8679f17
Lung_Opacity-3480.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3480.png,Lung_Opacity/images/Lung_Opacity-3480.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3480.png,Lung_Opacity/masks/Lung_Opacity-3480.png,True,False,299,299,L,113.44,f5c650d6a34666b6edfeba0d4212a861
Lung_Opacity-3481.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3481.png,Lung_Opacity/images/Lung_Opacity-3481.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3481.png,Lung_Opacity/masks/Lung_Opacity-3481.png,True,False,299,299,L,161.8,12c9c54a667268268013c1717ca1e32f
Lung_Opacity-3482.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3482.png,Lung_Opacity/images/Lung_Opacity-3482.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3482.png,Lung_Opacity/masks/Lung_Opacity-3482.png,True,False,299,299,L,118.28,efc8129ed77c551694a854171f9221e3
Lung_Opacity-3483.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3483.png,Lung_Opacity/images/Lung_Opacity-3483.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3483.png,Lung_Opacity/masks/Lung_Opacity-3483.png,True,False,299,299,L,106.79,367194d8e21b7e28caca593c0c13de90
Lung_Opacity-3484.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3484.png,Lung_Opacity/images/Lung_Opacity-3484.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3484.png,Lung_Opacity/masks/Lung_Opacity-3484.png,True,False,299,299,L,113.59,8913d85ea1b3d2046a786ee9b233960b
Lung_Opacity-3485.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3485.png,Lung_Opacity/images/Lung_Opacity-3485.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3485.png,Lung_Opacity/masks/Lung_Opacity-3485.png,True,False,299,299,L,122.08,60c3b28547ecad497726fcb8c1da3dad
Lung_Opacity-3486.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3486.png,Lung_Opacity/images/Lung_Opacity-3486.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3486.png,Lung_Opacity/masks/Lung_Opacity-3486.png,True,False,299,299,L,135.49,4f4f215a4b7f68d3473752667aeee499
Lung_Opacity-3487.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3487.png,Lung_Opacity/images/Lung_Opacity-3487.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3487.png,Lung_Opacity/masks/Lung_Opacity-3487.png,True,False,299,299,L,106.08,5cd0c442ec57a7dba52d8bf520eea29e
Lung_Opacity-3488.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3488.png,Lung_Opacity/images/Lung_Opacity-3488.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3488.png,Lung_Opacity/masks/Lung_Opacity-3488.png,True,False,299,299,L,161.07,c37bee0414ff6b4258bd009d6c19f535
Lung_Opacity-3489.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3489.png,Lung_Opacity/images/Lung_Opacity-3489.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3489.png,Lung_Opacity/masks/Lung_Opacity-3489.png,True,False,299,299,L,123.66,fed31c52c4c88611c41cb45488a92810
Lung_Opacity-349.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-349.png,Lung_Opacity/images/Lung_Opacity-349.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-349.png,Lung_Opacity/masks/Lung_Opacity-349.png,True,False,299,299,L,104.27,be4f6c6d5c6e856b7538fb539ac804b9
Lung_Opacity-3490.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3490.png,Lung_Opacity/images/Lung_Opacity-3490.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3490.png,Lung_Opacity/masks/Lung_Opacity-3490.png,True,False,299,299,L,119.01,e02ba023a9eda1eba96d43a448543e1c
Lung_Opacity-3491.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3491.png,Lung_Opacity/images/Lung_Opacity-3491.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3491.png,Lung_Opacity/masks/Lung_Opacity-3491.png,True,False,299,299,L,156.74,41f8438b9bc428dbc58afd46407c5f54
Lung_Opacity-3492.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3492.png,Lung_Opacity/images/Lung_Opacity-3492.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3492.png,Lung_Opacity/masks/Lung_Opacity-3492.png,True,False,299,299,L,112.75,c75203ab31ba0848bee113c5691a56b0
Lung_Opacity-3493.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3493.png,Lung_Opacity/images/Lung_Opacity-3493.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3493.png,Lung_Opacity/masks/Lung_Opacity-3493.png,True,False,299,299,L,115.49,5c7abe6fb8a5e2130d1e31b9430c629a
Lung_Opacity-3494.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3494.png,Lung_Opacity/images/Lung_Opacity-3494.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3494.png,Lung_Opacity/masks/Lung_Opacity-3494.png,True,False,299,299,L,131.2,89ff764dd571ac4f21c5672fa1f2cf1d
Lung_Opacity-3495.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3495.png,Lung_Opacity/images/Lung_Opacity-3495.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3495.png,Lung_Opacity/masks/Lung_Opacity-3495.png,True,False,299,299,L,115.27,dc588d931b9bf0ccfcd77da495848bd6
Lung_Opacity-3496.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3496.png,Lung_Opacity/images/Lung_Opacity-3496.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3496.png,Lung_Opacity/masks/Lung_Opacity-3496.png,True,False,299,299,L,149.1,6556d86ee7516230bd462a27f66f3b3f
Lung_Opacity-3497.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3497.png,Lung_Opacity/images/Lung_Opacity-3497.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3497.png,Lung_Opacity/masks/Lung_Opacity-3497.png,True,False,299,299,L,179.24,2e214e63a76d808a8cb7af5c107f2d95
Lung_Opacity-3498.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3498.png,Lung_Opacity/images/Lung_Opacity-3498.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3498.png,Lung_Opacity/masks/Lung_Opacity-3498.png,True,False,299,299,L,160.64,e1d739a6660a499650afafd2fa8466c1
Lung_Opacity-3499.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3499.png,Lung_Opacity/images/Lung_Opacity-3499.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3499.png,Lung_Opacity/masks/Lung_Opacity-3499.png,True,False,299,299,L,111.78,a5969a74845bd882f42bff65cb6b0bb0
Lung_Opacity-35.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-35.png,Lung_Opacity/images/Lung_Opacity-35.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-35.png,Lung_Opacity/masks/Lung_Opacity-35.png,True,False,299,299,L,114.23,3c6b15608d6882afa5a50be10dd6ba23
Lung_Opacity-350.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-350.png,Lung_Opacity/images/Lung_Opacity-350.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-350.png,Lung_Opacity/masks/Lung_Opacity-350.png,True,False,299,299,L,108.77,6690b3179a4608a2f436f00adee17e4a
Lung_Opacity-3500.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3500.png,Lung_Opacity/images/Lung_Opacity-3500.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3500.png,Lung_Opacity/masks/Lung_Opacity-3500.png,True,False,299,299,L,117.66,37a48051191738b8d5b3f7816051297f
Lung_Opacity-3501.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3501.png,Lung_Opacity/images/Lung_Opacity-3501.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3501.png,Lung_Opacity/masks/Lung_Opacity-3501.png,True,False,299,299,L,120.22,4c4248e662cdc67c93eb9f98a9d46f30
Lung_Opacity-3502.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3502.png,Lung_Opacity/images/Lung_Opacity-3502.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3502.png,Lung_Opacity/masks/Lung_Opacity-3502.png,True,False,299,299,L,156.67,6074b5453d2024b54888ff8df4b2ad26
Lung_Opacity-3503.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3503.png,Lung_Opacity/images/Lung_Opacity-3503.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3503.png,Lung_Opacity/masks/Lung_Opacity-3503.png,True,False,299,299,L,155.7,a7297702a9cd391045c43d62f23ca768
Lung_Opacity-3504.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3504.png,Lung_Opacity/images/Lung_Opacity-3504.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3504.png,Lung_Opacity/masks/Lung_Opacity-3504.png,True,False,299,299,L,120.99,568c6f12ebaa5a4db4df9c69f4eabdaa
Lung_Opacity-3505.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3505.png,Lung_Opacity/images/Lung_Opacity-3505.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3505.png,Lung_Opacity/masks/Lung_Opacity-3505.png,True,False,299,299,L,113.53,09e6a7973d18a253bc6cf4e934533c79
Lung_Opacity-3506.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3506.png,Lung_Opacity/images/Lung_Opacity-3506.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3506.png,Lung_Opacity/masks/Lung_Opacity-3506.png,True,False,299,299,L,129.87,7624bf86c505783bdcf416ea2a00fb84
Lung_Opacity-3507.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3507.png,Lung_Opacity/images/Lung_Opacity-3507.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3507.png,Lung_Opacity/masks/Lung_Opacity-3507.png,True,False,299,299,L,174.35,0bd96d6a2a67b3bb2b23d8916feef724
Lung_Opacity-3508.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3508.png,Lung_Opacity/images/Lung_Opacity-3508.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3508.png,Lung_Opacity/masks/Lung_Opacity-3508.png,True,False,299,299,L,150.91,fb476d48f570f508002dec87e2c9e649
Lung_Opacity-3509.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3509.png,Lung_Opacity/images/Lung_Opacity-3509.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3509.png,Lung_Opacity/masks/Lung_Opacity-3509.png,True,False,299,299,L,166.02,46699cf5be65a7362866245a8dbde2e7
Lung_Opacity-351.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-351.png,Lung_Opacity/images/Lung_Opacity-351.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-351.png,Lung_Opacity/masks/Lung_Opacity-351.png,True,False,299,299,L,112.37,804ae5ff5f0a695b446bc1dc18a321fa
Lung_Opacity-3510.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3510.png,Lung_Opacity/images/Lung_Opacity-3510.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3510.png,Lung_Opacity/masks/Lung_Opacity-3510.png,True,False,299,299,L,55.92,62aa57aa0b787a45952540ce3ac6bb7c
Lung_Opacity-3511.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3511.png,Lung_Opacity/images/Lung_Opacity-3511.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3511.png,Lung_Opacity/masks/Lung_Opacity-3511.png,True,False,299,299,L,132.42,53f7f0ba49e74eec4961f49f2e3fb2b9
Lung_Opacity-3512.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3512.png,Lung_Opacity/images/Lung_Opacity-3512.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3512.png,Lung_Opacity/masks/Lung_Opacity-3512.png,True,False,299,299,L,115.54,517013a47bfc930f595767d7d4835ae1
Lung_Opacity-3513.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3513.png,Lung_Opacity/images/Lung_Opacity-3513.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3513.png,Lung_Opacity/masks/Lung_Opacity-3513.png,True,False,299,299,L,159.28,788cf3d87fce5d7ad5de5b40c7421a38
Lung_Opacity-3514.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3514.png,Lung_Opacity/images/Lung_Opacity-3514.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3514.png,Lung_Opacity/masks/Lung_Opacity-3514.png,True,False,299,299,L,158.69,5ba1400d9b18cfd0e9d4ffc255a4466b
Lung_Opacity-3515.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3515.png,Lung_Opacity/images/Lung_Opacity-3515.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3515.png,Lung_Opacity/masks/Lung_Opacity-3515.png,True,False,299,299,L,125.32,77c4e94d24332c2c34aeb9626cdd03ea
Lung_Opacity-3516.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3516.png,Lung_Opacity/images/Lung_Opacity-3516.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3516.png,Lung_Opacity/masks/Lung_Opacity-3516.png,True,False,299,299,L,152.0,4a0a097b8d19dc98dc08ee2eba43994c
Lung_Opacity-3517.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3517.png,Lung_Opacity/images/Lung_Opacity-3517.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3517.png,Lung_Opacity/masks/Lung_Opacity-3517.png,True,False,299,299,L,114.44,ae982b391ba5b54a3a64b66c3fdcc31d
Lung_Opacity-3518.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3518.png,Lung_Opacity/images/Lung_Opacity-3518.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3518.png,Lung_Opacity/masks/Lung_Opacity-3518.png,True,False,299,299,L,144.86,5296e74ba3c8508559a804fd6b958822
Lung_Opacity-3519.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3519.png,Lung_Opacity/images/Lung_Opacity-3519.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3519.png,Lung_Opacity/masks/Lung_Opacity-3519.png,True,False,299,299,L,130.18,416f7f5b693d6e5f2b97433f2f81d45e
Lung_Opacity-352.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-352.png,Lung_Opacity/images/Lung_Opacity-352.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-352.png,Lung_Opacity/masks/Lung_Opacity-352.png,True,False,299,299,L,140.28,b8b16efa8d6b0136917e8f726f764463
Lung_Opacity-3520.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3520.png,Lung_Opacity/images/Lung_Opacity-3520.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3520.png,Lung_Opacity/masks/Lung_Opacity-3520.png,True,False,299,299,L,106.95,ed57cf7875ee14802d8f410522e4ac67
Lung_Opacity-3521.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3521.png,Lung_Opacity/images/Lung_Opacity-3521.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3521.png,Lung_Opacity/masks/Lung_Opacity-3521.png,True,False,299,299,L,115.09,04b638cb875d179fbe0b78a9916dc88b
Lung_Opacity-3522.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3522.png,Lung_Opacity/images/Lung_Opacity-3522.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3522.png,Lung_Opacity/masks/Lung_Opacity-3522.png,True,False,299,299,L,94.11,efc1dbd505bcd78496a53d1257ce99d1
Lung_Opacity-3523.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3523.png,Lung_Opacity/images/Lung_Opacity-3523.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3523.png,Lung_Opacity/masks/Lung_Opacity-3523.png,True,False,299,299,L,155.3,213889cb6ef98c8aaa2fac12a9949eae
Lung_Opacity-3524.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3524.png,Lung_Opacity/images/Lung_Opacity-3524.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3524.png,Lung_Opacity/masks/Lung_Opacity-3524.png,True,False,299,299,L,158.06,39edf1f094764042826f7dc4efef1fb2
Lung_Opacity-3525.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3525.png,Lung_Opacity/images/Lung_Opacity-3525.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3525.png,Lung_Opacity/masks/Lung_Opacity-3525.png,True,False,299,299,L,158.65,a81b9f138e1467a78ceb44979407ecf9
Lung_Opacity-3526.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3526.png,Lung_Opacity/images/Lung_Opacity-3526.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3526.png,Lung_Opacity/masks/Lung_Opacity-3526.png,True,False,299,299,L,128.73,25c65af238d2e07db13fd5148c594ad0
Lung_Opacity-3527.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3527.png,Lung_Opacity/images/Lung_Opacity-3527.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3527.png,Lung_Opacity/masks/Lung_Opacity-3527.png,True,False,299,299,L,158.94,c0ed282b81bc41acc9207da999a95d75
Lung_Opacity-3528.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3528.png,Lung_Opacity/images/Lung_Opacity-3528.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3528.png,Lung_Opacity/masks/Lung_Opacity-3528.png,True,False,299,299,L,137.23,7df54574a87a8695ad32d491ff4a0939
Lung_Opacity-3529.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3529.png,Lung_Opacity/images/Lung_Opacity-3529.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3529.png,Lung_Opacity/masks/Lung_Opacity-3529.png,True,False,299,299,L,145.87,d9ef7e0fe7a3a65a02056b63d70739bc
Lung_Opacity-353.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-353.png,Lung_Opacity/images/Lung_Opacity-353.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-353.png,Lung_Opacity/masks/Lung_Opacity-353.png,True,False,299,299,L,140.34,5d9da8ca113a4a367c8adc2a07a07807
Lung_Opacity-3530.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3530.png,Lung_Opacity/images/Lung_Opacity-3530.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3530.png,Lung_Opacity/masks/Lung_Opacity-3530.png,True,False,299,299,L,152.09,17195cc15c3ff71a713773f6b14829c6
Lung_Opacity-3531.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3531.png,Lung_Opacity/images/Lung_Opacity-3531.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3531.png,Lung_Opacity/masks/Lung_Opacity-3531.png,True,False,299,299,L,125.2,9c8d928a5cc9be89477dbbc75c19429c
Lung_Opacity-3532.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3532.png,Lung_Opacity/images/Lung_Opacity-3532.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3532.png,Lung_Opacity/masks/Lung_Opacity-3532.png,True,False,299,299,L,124.86,03edfe63ba3b580f382567ec0af890fe
Lung_Opacity-3533.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3533.png,Lung_Opacity/images/Lung_Opacity-3533.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3533.png,Lung_Opacity/masks/Lung_Opacity-3533.png,True,False,299,299,L,108.74,800902f08be5a46baa874b1fc7fbcc62
Lung_Opacity-3534.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3534.png,Lung_Opacity/images/Lung_Opacity-3534.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3534.png,Lung_Opacity/masks/Lung_Opacity-3534.png,True,False,299,299,L,156.72,cb9c2e5cc3dd8642ba85b2c7a391f85d
Lung_Opacity-3535.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3535.png,Lung_Opacity/images/Lung_Opacity-3535.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3535.png,Lung_Opacity/masks/Lung_Opacity-3535.png,True,False,299,299,L,99.44,5deee13f8d935249ba74d470a64ec8c1
Lung_Opacity-3536.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3536.png,Lung_Opacity/images/Lung_Opacity-3536.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3536.png,Lung_Opacity/masks/Lung_Opacity-3536.png,True,False,299,299,L,136.2,5b3af8604fbcc20922a672c3b8b58578
Lung_Opacity-3537.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3537.png,Lung_Opacity/images/Lung_Opacity-3537.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3537.png,Lung_Opacity/masks/Lung_Opacity-3537.png,True,False,299,299,L,109.12,60b09983e4ef9499f0bcbefe1643c015
Lung_Opacity-3538.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3538.png,Lung_Opacity/images/Lung_Opacity-3538.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3538.png,Lung_Opacity/masks/Lung_Opacity-3538.png,True,False,299,299,L,126.12,eee2889782ee790d662cec42217dd644
Lung_Opacity-3539.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3539.png,Lung_Opacity/images/Lung_Opacity-3539.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3539.png,Lung_Opacity/masks/Lung_Opacity-3539.png,True,False,299,299,L,107.82,496e796adc24fb881d2c0987a697298a
Lung_Opacity-354.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-354.png,Lung_Opacity/images/Lung_Opacity-354.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-354.png,Lung_Opacity/masks/Lung_Opacity-354.png,True,False,299,299,L,121.71,ace9816e25f274456a2da5accb90d71e
Lung_Opacity-3540.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3540.png,Lung_Opacity/images/Lung_Opacity-3540.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3540.png,Lung_Opacity/masks/Lung_Opacity-3540.png,True,False,299,299,L,141.06,d639101595296bcb60def78e1a701a6b
Lung_Opacity-3541.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3541.png,Lung_Opacity/images/Lung_Opacity-3541.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3541.png,Lung_Opacity/masks/Lung_Opacity-3541.png,True,False,299,299,L,169.39,c22756b9e0db09deb5e15e65a76442f9
Lung_Opacity-3542.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3542.png,Lung_Opacity/images/Lung_Opacity-3542.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3542.png,Lung_Opacity/masks/Lung_Opacity-3542.png,True,False,299,299,L,119.56,12769115aa3912aff739b2ad932c6fe5
Lung_Opacity-3543.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3543.png,Lung_Opacity/images/Lung_Opacity-3543.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3543.png,Lung_Opacity/masks/Lung_Opacity-3543.png,True,False,299,299,L,96.48,072221a9b1afe6c7304b78abe6b58597
Lung_Opacity-3544.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3544.png,Lung_Opacity/images/Lung_Opacity-3544.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3544.png,Lung_Opacity/masks/Lung_Opacity-3544.png,True,False,299,299,L,138.85,3b95d546c8682a3ee1592f34c046fb4c
Lung_Opacity-3545.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3545.png,Lung_Opacity/images/Lung_Opacity-3545.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3545.png,Lung_Opacity/masks/Lung_Opacity-3545.png,True,False,299,299,L,112.2,e672569ad1e1580bb2aa91b947f7d583
Lung_Opacity-3546.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3546.png,Lung_Opacity/images/Lung_Opacity-3546.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3546.png,Lung_Opacity/masks/Lung_Opacity-3546.png,True,False,299,299,L,135.78,4130dbf4be3f8f8f7d6d5189b2e723a8
Lung_Opacity-3547.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3547.png,Lung_Opacity/images/Lung_Opacity-3547.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3547.png,Lung_Opacity/masks/Lung_Opacity-3547.png,True,False,299,299,L,168.37,f5e9d054f82febfeb30fd2dde6ede792
Lung_Opacity-3548.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3548.png,Lung_Opacity/images/Lung_Opacity-3548.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3548.png,Lung_Opacity/masks/Lung_Opacity-3548.png,True,False,299,299,L,113.63,18e4ac4e23739a36fb35a9961cbd921f
Lung_Opacity-3549.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3549.png,Lung_Opacity/images/Lung_Opacity-3549.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3549.png,Lung_Opacity/masks/Lung_Opacity-3549.png,True,False,299,299,L,152.32,19e808803bfd29cab91b0ac51e63b1f7
Lung_Opacity-355.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-355.png,Lung_Opacity/images/Lung_Opacity-355.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-355.png,Lung_Opacity/masks/Lung_Opacity-355.png,True,False,299,299,L,154.54,57fd74bb7357ed71567a707054d499a6
Lung_Opacity-3550.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3550.png,Lung_Opacity/images/Lung_Opacity-3550.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3550.png,Lung_Opacity/masks/Lung_Opacity-3550.png,True,False,299,299,L,132.56,2c32992c0327d755da17ef5db0b876e4
Lung_Opacity-3551.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3551.png,Lung_Opacity/images/Lung_Opacity-3551.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3551.png,Lung_Opacity/masks/Lung_Opacity-3551.png,True,False,299,299,L,167.94,b48c362f41132a0b3e56e38c6a8f5d41
Lung_Opacity-3552.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3552.png,Lung_Opacity/images/Lung_Opacity-3552.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3552.png,Lung_Opacity/masks/Lung_Opacity-3552.png,True,False,299,299,L,121.15,1d3e894d9bb1f7b3d0e353f1d39d1612
Lung_Opacity-3553.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3553.png,Lung_Opacity/images/Lung_Opacity-3553.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3553.png,Lung_Opacity/masks/Lung_Opacity-3553.png,True,False,299,299,L,125.7,71827b242f4c84624ec49f2a37ff15df
Lung_Opacity-3554.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3554.png,Lung_Opacity/images/Lung_Opacity-3554.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3554.png,Lung_Opacity/masks/Lung_Opacity-3554.png,True,False,299,299,L,100.72,dfbdabe2d334af4f798dfbdfad571e15
Lung_Opacity-3555.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3555.png,Lung_Opacity/images/Lung_Opacity-3555.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3555.png,Lung_Opacity/masks/Lung_Opacity-3555.png,True,False,299,299,L,122.35,239fae9da6ac639a42e6d0e4eaf86dc4
Lung_Opacity-3556.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3556.png,Lung_Opacity/images/Lung_Opacity-3556.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3556.png,Lung_Opacity/masks/Lung_Opacity-3556.png,True,False,299,299,L,114.98,3bb5a2697c1ec00456349d329ffffe50
Lung_Opacity-3557.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3557.png,Lung_Opacity/images/Lung_Opacity-3557.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3557.png,Lung_Opacity/masks/Lung_Opacity-3557.png,True,False,299,299,L,108.14,7d4104ac47ec81933c39b27ec77edcfb
Lung_Opacity-3558.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3558.png,Lung_Opacity/images/Lung_Opacity-3558.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3558.png,Lung_Opacity/masks/Lung_Opacity-3558.png,True,False,299,299,L,113.33,b757be960eab0c369f9704bdb20d8bdd
Lung_Opacity-3559.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3559.png,Lung_Opacity/images/Lung_Opacity-3559.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3559.png,Lung_Opacity/masks/Lung_Opacity-3559.png,True,False,299,299,L,99.75,078790a111f9b961283e2870b9ea38ac
Lung_Opacity-356.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-356.png,Lung_Opacity/images/Lung_Opacity-356.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-356.png,Lung_Opacity/masks/Lung_Opacity-356.png,True,False,299,299,L,140.61,6b5fc3b7bd11cc5a8d8ae6149f0b9b02
Lung_Opacity-3560.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3560.png,Lung_Opacity/images/Lung_Opacity-3560.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3560.png,Lung_Opacity/masks/Lung_Opacity-3560.png,True,False,299,299,L,125.06,c622d0123b2eefed3cbeec32777ecf17
Lung_Opacity-3561.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3561.png,Lung_Opacity/images/Lung_Opacity-3561.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3561.png,Lung_Opacity/masks/Lung_Opacity-3561.png,True,False,299,299,L,161.36,4110a9ba9709d79ae9805c872c99b146
Lung_Opacity-3562.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3562.png,Lung_Opacity/images/Lung_Opacity-3562.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3562.png,Lung_Opacity/masks/Lung_Opacity-3562.png,True,False,299,299,L,121.33,dc160ef4400aecaaafc7495584ef994f
Lung_Opacity-3563.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3563.png,Lung_Opacity/images/Lung_Opacity-3563.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3563.png,Lung_Opacity/masks/Lung_Opacity-3563.png,True,False,299,299,L,146.9,5d80cb48c08241efbb175c1f213b044f
Lung_Opacity-3564.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3564.png,Lung_Opacity/images/Lung_Opacity-3564.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3564.png,Lung_Opacity/masks/Lung_Opacity-3564.png,True,False,299,299,L,118.29,6e5951a2490db2533170a09d613b1ec1
Lung_Opacity-3565.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3565.png,Lung_Opacity/images/Lung_Opacity-3565.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3565.png,Lung_Opacity/masks/Lung_Opacity-3565.png,True,False,299,299,L,116.59,13d1a756f89266ee781b6d7059ef0536
Lung_Opacity-3566.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3566.png,Lung_Opacity/images/Lung_Opacity-3566.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3566.png,Lung_Opacity/masks/Lung_Opacity-3566.png,True,False,299,299,L,152.23,d0a4d57b30f5eaacae271b99cf694c8d
Lung_Opacity-3567.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3567.png,Lung_Opacity/images/Lung_Opacity-3567.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3567.png,Lung_Opacity/masks/Lung_Opacity-3567.png,True,False,299,299,L,108.75,e2c26a2fe381b92f4d9a8c1c9f06a556
Lung_Opacity-3568.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3568.png,Lung_Opacity/images/Lung_Opacity-3568.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3568.png,Lung_Opacity/masks/Lung_Opacity-3568.png,True,False,299,299,L,160.05,f4e23bbf47281d6ec9cdf0a2bed74e1f
Lung_Opacity-3569.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3569.png,Lung_Opacity/images/Lung_Opacity-3569.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3569.png,Lung_Opacity/masks/Lung_Opacity-3569.png,True,False,299,299,L,116.5,f0be0bbe6915aff008109260eaf63232
Lung_Opacity-357.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-357.png,Lung_Opacity/images/Lung_Opacity-357.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-357.png,Lung_Opacity/masks/Lung_Opacity-357.png,True,False,299,299,L,125.76,383a9a0705b8eb3a76ba58fb74aec9a0
Lung_Opacity-3570.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3570.png,Lung_Opacity/images/Lung_Opacity-3570.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3570.png,Lung_Opacity/masks/Lung_Opacity-3570.png,True,False,299,299,L,107.41,ef948fc580b04fdff2c24b77ba0cf514
Lung_Opacity-3571.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3571.png,Lung_Opacity/images/Lung_Opacity-3571.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3571.png,Lung_Opacity/masks/Lung_Opacity-3571.png,True,False,299,299,L,127.19,1a63c5d627eb438c8e2e0f79b8a9135e
Lung_Opacity-3572.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3572.png,Lung_Opacity/images/Lung_Opacity-3572.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3572.png,Lung_Opacity/masks/Lung_Opacity-3572.png,True,False,299,299,L,100.84,5b24110ab38875fe9ccd69e29dcd65ac
Lung_Opacity-3573.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3573.png,Lung_Opacity/images/Lung_Opacity-3573.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3573.png,Lung_Opacity/masks/Lung_Opacity-3573.png,True,False,299,299,L,113.39,72b6c1d801e78bf6adcae749fd10065d
Lung_Opacity-3574.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3574.png,Lung_Opacity/images/Lung_Opacity-3574.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3574.png,Lung_Opacity/masks/Lung_Opacity-3574.png,True,False,299,299,L,161.22,15524f7afaa08a435e4f61b7ac68a90e
Lung_Opacity-3575.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3575.png,Lung_Opacity/images/Lung_Opacity-3575.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3575.png,Lung_Opacity/masks/Lung_Opacity-3575.png,True,False,299,299,L,84.49,b2940fa44788861ec9238072546d4ad8
Lung_Opacity-3576.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3576.png,Lung_Opacity/images/Lung_Opacity-3576.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3576.png,Lung_Opacity/masks/Lung_Opacity-3576.png,True,False,299,299,L,152.91,ca4f7a8d1bc4c11c307071fe8226d64c
Lung_Opacity-3577.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3577.png,Lung_Opacity/images/Lung_Opacity-3577.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3577.png,Lung_Opacity/masks/Lung_Opacity-3577.png,True,False,299,299,L,109.32,94f65626030e34cdb5f3f9fc0a3423f3
Lung_Opacity-3578.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3578.png,Lung_Opacity/images/Lung_Opacity-3578.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3578.png,Lung_Opacity/masks/Lung_Opacity-3578.png,True,False,299,299,L,132.85,38778e292afb88eb63d7cb3910a924ba
Lung_Opacity-3579.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3579.png,Lung_Opacity/images/Lung_Opacity-3579.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3579.png,Lung_Opacity/masks/Lung_Opacity-3579.png,True,False,299,299,L,90.72,54684fefe93d031e7793d91eebd8e109
Lung_Opacity-358.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-358.png,Lung_Opacity/images/Lung_Opacity-358.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-358.png,Lung_Opacity/masks/Lung_Opacity-358.png,True,False,299,299,L,154.98,219bd2bae22c1b72c8dac548cf2ac127
Lung_Opacity-3580.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3580.png,Lung_Opacity/images/Lung_Opacity-3580.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3580.png,Lung_Opacity/masks/Lung_Opacity-3580.png,True,False,299,299,L,129.35,b0f3719aaea418e13c239281e6aeaaab
Lung_Opacity-3581.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3581.png,Lung_Opacity/images/Lung_Opacity-3581.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3581.png,Lung_Opacity/masks/Lung_Opacity-3581.png,True,False,299,299,L,170.77,1c263eaa6316620384fd15ef7ba3da03
Lung_Opacity-3582.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3582.png,Lung_Opacity/images/Lung_Opacity-3582.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3582.png,Lung_Opacity/masks/Lung_Opacity-3582.png,True,False,299,299,L,104.63,5ba839cd1d875e1dfba277fe01ffad51
Lung_Opacity-3583.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3583.png,Lung_Opacity/images/Lung_Opacity-3583.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3583.png,Lung_Opacity/masks/Lung_Opacity-3583.png,True,False,299,299,L,133.0,b533f00b3787c119c0601a5b529224f1
Lung_Opacity-3584.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3584.png,Lung_Opacity/images/Lung_Opacity-3584.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3584.png,Lung_Opacity/masks/Lung_Opacity-3584.png,True,False,299,299,L,122.8,e247d9c584afa314cb4e3e731940579c
Lung_Opacity-3585.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3585.png,Lung_Opacity/images/Lung_Opacity-3585.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3585.png,Lung_Opacity/masks/Lung_Opacity-3585.png,True,False,299,299,L,133.76,f7847fdbdbc20f8c2ab3e47e7275052d
Lung_Opacity-3586.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3586.png,Lung_Opacity/images/Lung_Opacity-3586.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3586.png,Lung_Opacity/masks/Lung_Opacity-3586.png,True,False,299,299,L,119.36,405c325fcd81820c3a69abbb92c7a932
Lung_Opacity-3587.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3587.png,Lung_Opacity/images/Lung_Opacity-3587.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3587.png,Lung_Opacity/masks/Lung_Opacity-3587.png,True,False,299,299,L,133.89,767def447cd0745b53d3c635dd24a5e4
Lung_Opacity-3588.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3588.png,Lung_Opacity/images/Lung_Opacity-3588.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3588.png,Lung_Opacity/masks/Lung_Opacity-3588.png,True,False,299,299,L,117.77,d7cf70e014a9bf61b3b74275f9aef4a3
Lung_Opacity-3589.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3589.png,Lung_Opacity/images/Lung_Opacity-3589.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3589.png,Lung_Opacity/masks/Lung_Opacity-3589.png,True,False,299,299,L,110.66,b7fdf7cb8632476451a3df13144981ae
Lung_Opacity-359.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-359.png,Lung_Opacity/images/Lung_Opacity-359.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-359.png,Lung_Opacity/masks/Lung_Opacity-359.png,True,False,299,299,L,155.35,b82af22454c6112d4fa60aca7328b58e
Lung_Opacity-3590.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3590.png,Lung_Opacity/images/Lung_Opacity-3590.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3590.png,Lung_Opacity/masks/Lung_Opacity-3590.png,True,False,299,299,L,164.76,1d90b6cc091406efeacc8e858b08ac9d
Lung_Opacity-3591.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3591.png,Lung_Opacity/images/Lung_Opacity-3591.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3591.png,Lung_Opacity/masks/Lung_Opacity-3591.png,True,False,299,299,L,131.33,398ff503c30083daa3e2cf269547df52
Lung_Opacity-3592.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3592.png,Lung_Opacity/images/Lung_Opacity-3592.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3592.png,Lung_Opacity/masks/Lung_Opacity-3592.png,True,False,299,299,L,122.65,af44251cc710113d6529213f47ab28ee
Lung_Opacity-3593.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3593.png,Lung_Opacity/images/Lung_Opacity-3593.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3593.png,Lung_Opacity/masks/Lung_Opacity-3593.png,True,False,299,299,L,113.27,1f236bfafcb4307b4628a2ee06857321
Lung_Opacity-3594.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3594.png,Lung_Opacity/images/Lung_Opacity-3594.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3594.png,Lung_Opacity/masks/Lung_Opacity-3594.png,True,False,299,299,L,108.16,698e0d664e15842ac33bed1c1b30481a
Lung_Opacity-3595.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3595.png,Lung_Opacity/images/Lung_Opacity-3595.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3595.png,Lung_Opacity/masks/Lung_Opacity-3595.png,True,False,299,299,L,109.36,0592ea07941bcf0214a0d5710997a667
Lung_Opacity-3596.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3596.png,Lung_Opacity/images/Lung_Opacity-3596.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3596.png,Lung_Opacity/masks/Lung_Opacity-3596.png,True,False,299,299,L,122.68,1d6999468a33282a7ac67df48381acfd
Lung_Opacity-3597.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3597.png,Lung_Opacity/images/Lung_Opacity-3597.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3597.png,Lung_Opacity/masks/Lung_Opacity-3597.png,True,False,299,299,L,165.16,df995ae5b7d0a20000445797e2da5e11
Lung_Opacity-3598.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3598.png,Lung_Opacity/images/Lung_Opacity-3598.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3598.png,Lung_Opacity/masks/Lung_Opacity-3598.png,True,False,299,299,L,118.37,99c8f466f9c7ac82c87e433f6d0e78ed
Lung_Opacity-3599.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3599.png,Lung_Opacity/images/Lung_Opacity-3599.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3599.png,Lung_Opacity/masks/Lung_Opacity-3599.png,True,False,299,299,L,116.71,92223bfc8f454a8662526806548d055e
Lung_Opacity-36.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-36.png,Lung_Opacity/images/Lung_Opacity-36.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-36.png,Lung_Opacity/masks/Lung_Opacity-36.png,True,False,299,299,L,108.49,80342d9e0a48e24f7aad0c9a4c9faed9
Lung_Opacity-360.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-360.png,Lung_Opacity/images/Lung_Opacity-360.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-360.png,Lung_Opacity/masks/Lung_Opacity-360.png,True,False,299,299,L,112.61,1bfe03e0cb4437f58482ea4eea73c3d4
Lung_Opacity-3600.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3600.png,Lung_Opacity/images/Lung_Opacity-3600.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3600.png,Lung_Opacity/masks/Lung_Opacity-3600.png,True,False,299,299,L,114.14,efc9bb7c3418eefb06f140f9bb1ee24f
Lung_Opacity-3601.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3601.png,Lung_Opacity/images/Lung_Opacity-3601.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3601.png,Lung_Opacity/masks/Lung_Opacity-3601.png,True,False,299,299,L,128.02,e4c4d3b3ccdf5e6f741af8607a66a35f
Lung_Opacity-3602.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3602.png,Lung_Opacity/images/Lung_Opacity-3602.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3602.png,Lung_Opacity/masks/Lung_Opacity-3602.png,True,False,299,299,L,113.71,5c177854f64c1437210ac5416f80fe1a
Lung_Opacity-3603.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3603.png,Lung_Opacity/images/Lung_Opacity-3603.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3603.png,Lung_Opacity/masks/Lung_Opacity-3603.png,True,False,299,299,L,114.94,b0e596a2fb32eb88c0b843194e3d7022
Lung_Opacity-3604.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3604.png,Lung_Opacity/images/Lung_Opacity-3604.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3604.png,Lung_Opacity/masks/Lung_Opacity-3604.png,True,False,299,299,L,163.33,45da636bcf1c6c36aaa118346fd96d7e
Lung_Opacity-3605.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3605.png,Lung_Opacity/images/Lung_Opacity-3605.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3605.png,Lung_Opacity/masks/Lung_Opacity-3605.png,True,False,299,299,L,121.07,42fc85ef1bff2fd4a7a4a05b0aab8d38
Lung_Opacity-3606.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3606.png,Lung_Opacity/images/Lung_Opacity-3606.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3606.png,Lung_Opacity/masks/Lung_Opacity-3606.png,True,False,299,299,L,110.2,3d336aee91c440f466443783625f5cef
Lung_Opacity-3607.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3607.png,Lung_Opacity/images/Lung_Opacity-3607.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3607.png,Lung_Opacity/masks/Lung_Opacity-3607.png,True,False,299,299,L,116.09,1419486fbffd978a811d32fdb0558690
Lung_Opacity-3608.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3608.png,Lung_Opacity/images/Lung_Opacity-3608.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3608.png,Lung_Opacity/masks/Lung_Opacity-3608.png,True,False,299,299,L,132.48,7bf563f3047e42db7dbce1e7b8ccd4c3
Lung_Opacity-3609.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3609.png,Lung_Opacity/images/Lung_Opacity-3609.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3609.png,Lung_Opacity/masks/Lung_Opacity-3609.png,True,False,299,299,L,108.55,a42d6ef63ae019e399883b49d3b895bd
Lung_Opacity-361.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-361.png,Lung_Opacity/images/Lung_Opacity-361.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-361.png,Lung_Opacity/masks/Lung_Opacity-361.png,True,False,299,299,L,135.91,c566c4de9cbb838de72903057f1d6751
Lung_Opacity-3610.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3610.png,Lung_Opacity/images/Lung_Opacity-3610.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3610.png,Lung_Opacity/masks/Lung_Opacity-3610.png,True,False,299,299,L,141.21,940342c33a7c5a8148cfae24da98968c
Lung_Opacity-3611.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3611.png,Lung_Opacity/images/Lung_Opacity-3611.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3611.png,Lung_Opacity/masks/Lung_Opacity-3611.png,True,False,299,299,L,125.9,e637f0799225c79936bf0181bcd28256
Lung_Opacity-3612.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3612.png,Lung_Opacity/images/Lung_Opacity-3612.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3612.png,Lung_Opacity/masks/Lung_Opacity-3612.png,True,False,299,299,L,132.72,4277ce1ed090f752c26124dfd2fb2fc2
Lung_Opacity-3613.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3613.png,Lung_Opacity/images/Lung_Opacity-3613.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3613.png,Lung_Opacity/masks/Lung_Opacity-3613.png,True,False,299,299,L,126.58,38dc9037a8dcde32802abc6fc3e0c26d
Lung_Opacity-3614.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3614.png,Lung_Opacity/images/Lung_Opacity-3614.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3614.png,Lung_Opacity/masks/Lung_Opacity-3614.png,True,False,299,299,L,51.14,4cafaeb1ac13a010ff50a3f15321c821
Lung_Opacity-3615.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3615.png,Lung_Opacity/images/Lung_Opacity-3615.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3615.png,Lung_Opacity/masks/Lung_Opacity-3615.png,True,False,299,299,L,106.69,d4ad5dec8b2504514f0c6b15fba73620
Lung_Opacity-3616.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3616.png,Lung_Opacity/images/Lung_Opacity-3616.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3616.png,Lung_Opacity/masks/Lung_Opacity-3616.png,True,False,299,299,L,131.9,a4caf8502e914cb34f2a6ed7344664e2
Lung_Opacity-3617.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3617.png,Lung_Opacity/images/Lung_Opacity-3617.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3617.png,Lung_Opacity/masks/Lung_Opacity-3617.png,True,False,299,299,L,165.36,c8921efe801b99a9171ed399844b2349
Lung_Opacity-3618.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3618.png,Lung_Opacity/images/Lung_Opacity-3618.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3618.png,Lung_Opacity/masks/Lung_Opacity-3618.png,True,False,299,299,L,180.03,789a2753a44587d2b03e6ee25fd1a53a
Lung_Opacity-3619.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3619.png,Lung_Opacity/images/Lung_Opacity-3619.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3619.png,Lung_Opacity/masks/Lung_Opacity-3619.png,True,False,299,299,L,98.22,11cbb99826fa9e5bbe8372c68f2c77a6
Lung_Opacity-362.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-362.png,Lung_Opacity/images/Lung_Opacity-362.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-362.png,Lung_Opacity/masks/Lung_Opacity-362.png,True,False,299,299,L,164.57,528d11c54b51e66ed11b8c240c90dc56
Lung_Opacity-3620.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3620.png,Lung_Opacity/images/Lung_Opacity-3620.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3620.png,Lung_Opacity/masks/Lung_Opacity-3620.png,True,False,299,299,L,116.96,0f01acb86b49f370436af3ba70a0d002
Lung_Opacity-3621.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3621.png,Lung_Opacity/images/Lung_Opacity-3621.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3621.png,Lung_Opacity/masks/Lung_Opacity-3621.png,True,False,299,299,L,160.66,e7efaf8a8e1a480fb0104cf3d6b9437a
Lung_Opacity-3622.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3622.png,Lung_Opacity/images/Lung_Opacity-3622.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3622.png,Lung_Opacity/masks/Lung_Opacity-3622.png,True,False,299,299,L,95.57,181d2be5b8105195b66a760d152c2caa
Lung_Opacity-3623.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3623.png,Lung_Opacity/images/Lung_Opacity-3623.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3623.png,Lung_Opacity/masks/Lung_Opacity-3623.png,True,False,299,299,L,106.26,9f5f5050047980d9dd2bd543793af326
Lung_Opacity-3624.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3624.png,Lung_Opacity/images/Lung_Opacity-3624.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3624.png,Lung_Opacity/masks/Lung_Opacity-3624.png,True,False,299,299,L,150.85,8aabd0e482c3e43cc3d7919ceae3fadf
Lung_Opacity-3625.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3625.png,Lung_Opacity/images/Lung_Opacity-3625.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3625.png,Lung_Opacity/masks/Lung_Opacity-3625.png,True,False,299,299,L,135.67,5af942f7315c1d9da11c2011aafe1929
Lung_Opacity-3626.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3626.png,Lung_Opacity/images/Lung_Opacity-3626.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3626.png,Lung_Opacity/masks/Lung_Opacity-3626.png,True,False,299,299,L,111.99,89b9ef8ce8b2e13ae3c4136e36e2c778
Lung_Opacity-3627.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3627.png,Lung_Opacity/images/Lung_Opacity-3627.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3627.png,Lung_Opacity/masks/Lung_Opacity-3627.png,True,False,299,299,L,117.12,13ac68a82e65cd83e2d7792ac5e7b05d
Lung_Opacity-3628.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3628.png,Lung_Opacity/images/Lung_Opacity-3628.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3628.png,Lung_Opacity/masks/Lung_Opacity-3628.png,True,False,299,299,L,96.24,f838bdf83a8d57b8f45fdbb12385df77
Lung_Opacity-3629.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3629.png,Lung_Opacity/images/Lung_Opacity-3629.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3629.png,Lung_Opacity/masks/Lung_Opacity-3629.png,True,False,299,299,L,130.95,f6aafb84ca673512861ad3c8f5b8b66e
Lung_Opacity-363.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-363.png,Lung_Opacity/images/Lung_Opacity-363.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-363.png,Lung_Opacity/masks/Lung_Opacity-363.png,True,False,299,299,L,114.79,dcc44259d4fa2bf62dbf32881a91eea8
Lung_Opacity-3630.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3630.png,Lung_Opacity/images/Lung_Opacity-3630.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3630.png,Lung_Opacity/masks/Lung_Opacity-3630.png,True,False,299,299,L,35.03,0d388a505f8e77a404a85075aeac977a
Lung_Opacity-3631.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3631.png,Lung_Opacity/images/Lung_Opacity-3631.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3631.png,Lung_Opacity/masks/Lung_Opacity-3631.png,True,False,299,299,L,105.89,a1f45fa1605e869a03f849dc521df6ed
Lung_Opacity-3632.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3632.png,Lung_Opacity/images/Lung_Opacity-3632.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3632.png,Lung_Opacity/masks/Lung_Opacity-3632.png,True,False,299,299,L,115.81,ac7e8ec1a881e01e55017324a76a3d0e
Lung_Opacity-3633.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3633.png,Lung_Opacity/images/Lung_Opacity-3633.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3633.png,Lung_Opacity/masks/Lung_Opacity-3633.png,True,False,299,299,L,115.3,f0e358b628f9169c1021fb4cc8d6d58a
Lung_Opacity-3634.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3634.png,Lung_Opacity/images/Lung_Opacity-3634.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3634.png,Lung_Opacity/masks/Lung_Opacity-3634.png,True,False,299,299,L,117.43,c3e4d75cb9377c3add31da9fb800f8a3
Lung_Opacity-3635.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3635.png,Lung_Opacity/images/Lung_Opacity-3635.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3635.png,Lung_Opacity/masks/Lung_Opacity-3635.png,True,False,299,299,L,106.2,47af1d66aba8b8f4b19b023ececefa8b
Lung_Opacity-3636.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3636.png,Lung_Opacity/images/Lung_Opacity-3636.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3636.png,Lung_Opacity/masks/Lung_Opacity-3636.png,True,False,299,299,L,146.17,fb2067f824470948f04cd7e755de808e
Lung_Opacity-3637.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3637.png,Lung_Opacity/images/Lung_Opacity-3637.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3637.png,Lung_Opacity/masks/Lung_Opacity-3637.png,True,False,299,299,L,116.09,8f6f1e8054478595889b8f410a1c979b
Lung_Opacity-3638.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3638.png,Lung_Opacity/images/Lung_Opacity-3638.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3638.png,Lung_Opacity/masks/Lung_Opacity-3638.png,True,False,299,299,L,130.79,a7e6878ee41340afe0d3f911f8b05754
Lung_Opacity-3639.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3639.png,Lung_Opacity/images/Lung_Opacity-3639.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3639.png,Lung_Opacity/masks/Lung_Opacity-3639.png,True,False,299,299,L,150.68,f955283f41db86786f921c73033a3da8
Lung_Opacity-364.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-364.png,Lung_Opacity/images/Lung_Opacity-364.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-364.png,Lung_Opacity/masks/Lung_Opacity-364.png,True,False,299,299,L,115.44,55a583ea4d69cdb1cbfcd35e71eac2e3
Lung_Opacity-3640.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3640.png,Lung_Opacity/images/Lung_Opacity-3640.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3640.png,Lung_Opacity/masks/Lung_Opacity-3640.png,True,False,299,299,L,130.19,e71b025d30e6b511133c8c43de7e3d0a
Lung_Opacity-3641.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3641.png,Lung_Opacity/images/Lung_Opacity-3641.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3641.png,Lung_Opacity/masks/Lung_Opacity-3641.png,True,False,299,299,L,91.73,265dfc7560fe523e444e3dea5d7d7086
Lung_Opacity-3642.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3642.png,Lung_Opacity/images/Lung_Opacity-3642.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3642.png,Lung_Opacity/masks/Lung_Opacity-3642.png,True,False,299,299,L,96.15,f64a1a979958e74ad795f26cd7e593d1
Lung_Opacity-3643.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3643.png,Lung_Opacity/images/Lung_Opacity-3643.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3643.png,Lung_Opacity/masks/Lung_Opacity-3643.png,True,False,299,299,L,153.97,7cec715851a3bf5edc6c50205b1f6d9c
Lung_Opacity-3644.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3644.png,Lung_Opacity/images/Lung_Opacity-3644.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3644.png,Lung_Opacity/masks/Lung_Opacity-3644.png,True,False,299,299,L,119.81,59785704972bb3d89d5621278bcf0f45
Lung_Opacity-3645.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3645.png,Lung_Opacity/images/Lung_Opacity-3645.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3645.png,Lung_Opacity/masks/Lung_Opacity-3645.png,True,False,299,299,L,101.96,6fcc8bfce23c0611c199fcf8c69fcc7f
Lung_Opacity-3646.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3646.png,Lung_Opacity/images/Lung_Opacity-3646.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3646.png,Lung_Opacity/masks/Lung_Opacity-3646.png,True,False,299,299,L,119.7,e77567be0f2a5762434f8ddafaaae3ce
Lung_Opacity-3647.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3647.png,Lung_Opacity/images/Lung_Opacity-3647.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3647.png,Lung_Opacity/masks/Lung_Opacity-3647.png,True,False,299,299,L,110.04,89b08dc6fd289b062bb06e902cd7249b
Lung_Opacity-3648.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3648.png,Lung_Opacity/images/Lung_Opacity-3648.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3648.png,Lung_Opacity/masks/Lung_Opacity-3648.png,True,False,299,299,L,163.86,b6a69d06200fc028243062011ff4b534
Lung_Opacity-3649.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3649.png,Lung_Opacity/images/Lung_Opacity-3649.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3649.png,Lung_Opacity/masks/Lung_Opacity-3649.png,True,False,299,299,L,126.89,bf5104d454a85d11dae436c4866a43ca
Lung_Opacity-365.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-365.png,Lung_Opacity/images/Lung_Opacity-365.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-365.png,Lung_Opacity/masks/Lung_Opacity-365.png,True,False,299,299,L,114.53,0d9f8aac4cce4df3dc657e943840f6d8
Lung_Opacity-3650.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3650.png,Lung_Opacity/images/Lung_Opacity-3650.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3650.png,Lung_Opacity/masks/Lung_Opacity-3650.png,True,False,299,299,L,128.48,d595e980a7671a921495a372d2d47f69
Lung_Opacity-3651.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3651.png,Lung_Opacity/images/Lung_Opacity-3651.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3651.png,Lung_Opacity/masks/Lung_Opacity-3651.png,True,False,299,299,L,105.76,aeee06b0dc7998e31caf160a3d6ea01c
Lung_Opacity-3652.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3652.png,Lung_Opacity/images/Lung_Opacity-3652.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3652.png,Lung_Opacity/masks/Lung_Opacity-3652.png,True,False,299,299,L,113.83,722b038939d9210dc27c2f8a11eb9fdb
Lung_Opacity-3653.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3653.png,Lung_Opacity/images/Lung_Opacity-3653.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3653.png,Lung_Opacity/masks/Lung_Opacity-3653.png,True,False,299,299,L,104.27,84de422fa7683640005c4bc2b86c5d01
Lung_Opacity-3654.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3654.png,Lung_Opacity/images/Lung_Opacity-3654.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3654.png,Lung_Opacity/masks/Lung_Opacity-3654.png,True,False,299,299,L,104.04,1d1dbc32dec9d5d9b51bb5ab7dd3ea97
Lung_Opacity-3655.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3655.png,Lung_Opacity/images/Lung_Opacity-3655.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3655.png,Lung_Opacity/masks/Lung_Opacity-3655.png,True,False,299,299,L,96.42,dd47e160d48cfa33a12eb479d2e7dfbf
Lung_Opacity-3656.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3656.png,Lung_Opacity/images/Lung_Opacity-3656.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3656.png,Lung_Opacity/masks/Lung_Opacity-3656.png,True,False,299,299,L,109.14,af9189454ad4df3f5d7776bbeb9f6961
Lung_Opacity-3657.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3657.png,Lung_Opacity/images/Lung_Opacity-3657.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3657.png,Lung_Opacity/masks/Lung_Opacity-3657.png,True,False,299,299,L,144.19,1dbfc290527b1cd77e92906a0976acab
Lung_Opacity-3658.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3658.png,Lung_Opacity/images/Lung_Opacity-3658.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3658.png,Lung_Opacity/masks/Lung_Opacity-3658.png,True,False,299,299,L,112.78,d8250aa78c89fa60e32fd76f8565884b
Lung_Opacity-3659.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3659.png,Lung_Opacity/images/Lung_Opacity-3659.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3659.png,Lung_Opacity/masks/Lung_Opacity-3659.png,True,False,299,299,L,121.95,8e57f0fed0c60d5858394757cb6fcad7
Lung_Opacity-366.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-366.png,Lung_Opacity/images/Lung_Opacity-366.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-366.png,Lung_Opacity/masks/Lung_Opacity-366.png,True,False,299,299,L,120.63,2692c12cb624e5f6ca4683816daecd95
Lung_Opacity-3660.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3660.png,Lung_Opacity/images/Lung_Opacity-3660.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3660.png,Lung_Opacity/masks/Lung_Opacity-3660.png,True,False,299,299,L,122.47,d25ded3d118c32f32e5aeba92ace7521
Lung_Opacity-3661.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3661.png,Lung_Opacity/images/Lung_Opacity-3661.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3661.png,Lung_Opacity/masks/Lung_Opacity-3661.png,True,False,299,299,L,118.05,b4cd62a867b17f256ea14a59b967b8ef
Lung_Opacity-3662.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3662.png,Lung_Opacity/images/Lung_Opacity-3662.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3662.png,Lung_Opacity/masks/Lung_Opacity-3662.png,True,False,299,299,L,145.14,8cfe2fedd29c09ac2a71ae9d47c67293
Lung_Opacity-3663.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3663.png,Lung_Opacity/images/Lung_Opacity-3663.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3663.png,Lung_Opacity/masks/Lung_Opacity-3663.png,True,False,299,299,L,111.58,fd574fe1c4a5507a296caf248daac8b3
Lung_Opacity-3664.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3664.png,Lung_Opacity/images/Lung_Opacity-3664.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3664.png,Lung_Opacity/masks/Lung_Opacity-3664.png,True,False,299,299,L,110.97,d6d691e6cca037f534867eea5413124d
Lung_Opacity-3665.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3665.png,Lung_Opacity/images/Lung_Opacity-3665.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3665.png,Lung_Opacity/masks/Lung_Opacity-3665.png,True,False,299,299,L,87.94,c09ca42bea733694fa68f14cc001d924
Lung_Opacity-3666.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3666.png,Lung_Opacity/images/Lung_Opacity-3666.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3666.png,Lung_Opacity/masks/Lung_Opacity-3666.png,True,False,299,299,L,117.85,834f280040126a38dc3288048a79fc61
Lung_Opacity-3667.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3667.png,Lung_Opacity/images/Lung_Opacity-3667.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3667.png,Lung_Opacity/masks/Lung_Opacity-3667.png,True,False,299,299,L,116.93,4b91bbd6d5b5aa579857153b1e88eb76
Lung_Opacity-3668.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3668.png,Lung_Opacity/images/Lung_Opacity-3668.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3668.png,Lung_Opacity/masks/Lung_Opacity-3668.png,True,False,299,299,L,131.26,af16a0d4efe0a853e3382628613c5aff
Lung_Opacity-3669.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3669.png,Lung_Opacity/images/Lung_Opacity-3669.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3669.png,Lung_Opacity/masks/Lung_Opacity-3669.png,True,False,299,299,L,139.92,074f49c90fab82f5054e3bce80113b1e
Lung_Opacity-367.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-367.png,Lung_Opacity/images/Lung_Opacity-367.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-367.png,Lung_Opacity/masks/Lung_Opacity-367.png,True,False,299,299,L,154.26,7b663839902632dca2acaf1c776ce634
Lung_Opacity-3670.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3670.png,Lung_Opacity/images/Lung_Opacity-3670.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3670.png,Lung_Opacity/masks/Lung_Opacity-3670.png,True,False,299,299,L,111.92,1c9b4127d5114aab44c75c823d881d4d
Lung_Opacity-3671.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3671.png,Lung_Opacity/images/Lung_Opacity-3671.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3671.png,Lung_Opacity/masks/Lung_Opacity-3671.png,True,False,299,299,L,135.19,db1e9e47da7e13addb370afb5b65a1fb
Lung_Opacity-3672.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3672.png,Lung_Opacity/images/Lung_Opacity-3672.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3672.png,Lung_Opacity/masks/Lung_Opacity-3672.png,True,False,299,299,L,110.15,97d1c1aa0bda7e794766c4f95070680e
Lung_Opacity-3673.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3673.png,Lung_Opacity/images/Lung_Opacity-3673.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3673.png,Lung_Opacity/masks/Lung_Opacity-3673.png,True,False,299,299,L,121.74,e99ff63f773061b130a6d93cfa59b40f
Lung_Opacity-3674.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3674.png,Lung_Opacity/images/Lung_Opacity-3674.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3674.png,Lung_Opacity/masks/Lung_Opacity-3674.png,True,False,299,299,L,160.01,17c979ef29b15fea3541851ae79938d4
Lung_Opacity-3675.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3675.png,Lung_Opacity/images/Lung_Opacity-3675.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3675.png,Lung_Opacity/masks/Lung_Opacity-3675.png,True,False,299,299,L,122.99,b10bd697554e63b466ac545da3f8549f
Lung_Opacity-3676.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3676.png,Lung_Opacity/images/Lung_Opacity-3676.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3676.png,Lung_Opacity/masks/Lung_Opacity-3676.png,True,False,299,299,L,93.63,0276fff0ad0c4b76aaf9696b0cef2473
Lung_Opacity-3677.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3677.png,Lung_Opacity/images/Lung_Opacity-3677.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3677.png,Lung_Opacity/masks/Lung_Opacity-3677.png,True,False,299,299,L,111.54,780e87f98d632d52a2eb24f0d679c4da
Lung_Opacity-3678.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3678.png,Lung_Opacity/images/Lung_Opacity-3678.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3678.png,Lung_Opacity/masks/Lung_Opacity-3678.png,True,False,299,299,L,117.13,31502a8349305888e86aba4030c0b0ea
Lung_Opacity-3679.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3679.png,Lung_Opacity/images/Lung_Opacity-3679.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3679.png,Lung_Opacity/masks/Lung_Opacity-3679.png,True,False,299,299,L,165.93,4f99e1dd492763523cec645cd0df5fc3
Lung_Opacity-368.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-368.png,Lung_Opacity/images/Lung_Opacity-368.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-368.png,Lung_Opacity/masks/Lung_Opacity-368.png,True,False,299,299,L,162.35,e952fc154e26f612298c6f7b87cf541e
Lung_Opacity-3680.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3680.png,Lung_Opacity/images/Lung_Opacity-3680.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3680.png,Lung_Opacity/masks/Lung_Opacity-3680.png,True,False,299,299,L,109.38,dc3fe24878524b03b7e4999e090c3757
Lung_Opacity-3681.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3681.png,Lung_Opacity/images/Lung_Opacity-3681.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3681.png,Lung_Opacity/masks/Lung_Opacity-3681.png,True,False,299,299,L,130.84,6fcd85a933de7182552fadcedb1583e6
Lung_Opacity-3682.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3682.png,Lung_Opacity/images/Lung_Opacity-3682.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3682.png,Lung_Opacity/masks/Lung_Opacity-3682.png,True,False,299,299,L,103.45,5e9a154800a27cd86763abc14522b416
Lung_Opacity-3683.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3683.png,Lung_Opacity/images/Lung_Opacity-3683.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3683.png,Lung_Opacity/masks/Lung_Opacity-3683.png,True,False,299,299,L,175.24,e1f04e1751f096be3c87e50399119f6e
Lung_Opacity-3684.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3684.png,Lung_Opacity/images/Lung_Opacity-3684.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3684.png,Lung_Opacity/masks/Lung_Opacity-3684.png,True,False,299,299,L,141.33,9796deadcb98af0852a8434af9933f1d
Lung_Opacity-3685.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3685.png,Lung_Opacity/images/Lung_Opacity-3685.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3685.png,Lung_Opacity/masks/Lung_Opacity-3685.png,True,False,299,299,L,106.77,09cab2dec054eb8b60a30119f4520ed9
Lung_Opacity-3686.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3686.png,Lung_Opacity/images/Lung_Opacity-3686.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3686.png,Lung_Opacity/masks/Lung_Opacity-3686.png,True,False,299,299,L,108.46,038ffdfe3e3b1e628127a3526b395320
Lung_Opacity-3687.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3687.png,Lung_Opacity/images/Lung_Opacity-3687.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3687.png,Lung_Opacity/masks/Lung_Opacity-3687.png,True,False,299,299,L,146.14,dc09c5e2179e8dbb7b95187569c0e6da
Lung_Opacity-3688.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3688.png,Lung_Opacity/images/Lung_Opacity-3688.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3688.png,Lung_Opacity/masks/Lung_Opacity-3688.png,True,False,299,299,L,110.83,dc3807b62494cb8b45c82e4d244af0f8
Lung_Opacity-3689.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3689.png,Lung_Opacity/images/Lung_Opacity-3689.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3689.png,Lung_Opacity/masks/Lung_Opacity-3689.png,True,False,299,299,L,102.43,6b93f688b8ae363164d4ff3de9c4466d
Lung_Opacity-369.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-369.png,Lung_Opacity/images/Lung_Opacity-369.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-369.png,Lung_Opacity/masks/Lung_Opacity-369.png,True,False,299,299,L,156.5,84df66791827c762b2e3350db9dd27b6
Lung_Opacity-3690.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3690.png,Lung_Opacity/images/Lung_Opacity-3690.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3690.png,Lung_Opacity/masks/Lung_Opacity-3690.png,True,False,299,299,L,106.53,4bd62330357d833a7f1678dbe852815d
Lung_Opacity-3691.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3691.png,Lung_Opacity/images/Lung_Opacity-3691.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3691.png,Lung_Opacity/masks/Lung_Opacity-3691.png,True,False,299,299,L,102.49,d2ffb019de4120f5d8758952847d6a55
Lung_Opacity-3692.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3692.png,Lung_Opacity/images/Lung_Opacity-3692.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3692.png,Lung_Opacity/masks/Lung_Opacity-3692.png,True,False,299,299,L,98.49,e5e657b6f30bd1976a2c693d370e9acf
Lung_Opacity-3693.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3693.png,Lung_Opacity/images/Lung_Opacity-3693.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3693.png,Lung_Opacity/masks/Lung_Opacity-3693.png,True,False,299,299,L,117.75,317dc7b44be2608cb030a47f51ceac8d
Lung_Opacity-3694.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3694.png,Lung_Opacity/images/Lung_Opacity-3694.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3694.png,Lung_Opacity/masks/Lung_Opacity-3694.png,True,False,299,299,L,102.25,53869cc27b915de58a7f215cd776b1e6
Lung_Opacity-3695.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3695.png,Lung_Opacity/images/Lung_Opacity-3695.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3695.png,Lung_Opacity/masks/Lung_Opacity-3695.png,True,False,299,299,L,154.51,5e70b8e20811d51abf285c43baabdf84
Lung_Opacity-3696.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3696.png,Lung_Opacity/images/Lung_Opacity-3696.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3696.png,Lung_Opacity/masks/Lung_Opacity-3696.png,True,False,299,299,L,133.52,c20f4bfb2c4ccfa33b79513a25b7f0e7
Lung_Opacity-3697.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3697.png,Lung_Opacity/images/Lung_Opacity-3697.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3697.png,Lung_Opacity/masks/Lung_Opacity-3697.png,True,False,299,299,L,130.8,f34df9e046e1762f8d73260aacaf8b02
Lung_Opacity-3698.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3698.png,Lung_Opacity/images/Lung_Opacity-3698.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3698.png,Lung_Opacity/masks/Lung_Opacity-3698.png,True,False,299,299,L,125.31,4c50facc2d16db209259d24adee5d4a8
Lung_Opacity-3699.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3699.png,Lung_Opacity/images/Lung_Opacity-3699.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3699.png,Lung_Opacity/masks/Lung_Opacity-3699.png,True,False,299,299,L,164.32,f98472dfc4a830a317ef528229640ae6
Lung_Opacity-37.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-37.png,Lung_Opacity/images/Lung_Opacity-37.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-37.png,Lung_Opacity/masks/Lung_Opacity-37.png,True,False,299,299,L,169.44,0f5657c51058a66ca8f0283f9f75e383
Lung_Opacity-370.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-370.png,Lung_Opacity/images/Lung_Opacity-370.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-370.png,Lung_Opacity/masks/Lung_Opacity-370.png,True,False,299,299,L,177.78,b97decfe6e204925f4cc5cca2bb683a7
Lung_Opacity-3700.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3700.png,Lung_Opacity/images/Lung_Opacity-3700.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3700.png,Lung_Opacity/masks/Lung_Opacity-3700.png,True,False,299,299,L,91.4,33aebe5e30a1b2c0908fa41aae256381
Lung_Opacity-3701.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3701.png,Lung_Opacity/images/Lung_Opacity-3701.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3701.png,Lung_Opacity/masks/Lung_Opacity-3701.png,True,False,299,299,L,155.56,66986a62797465251ea856fdcc9b05af
Lung_Opacity-3702.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3702.png,Lung_Opacity/images/Lung_Opacity-3702.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3702.png,Lung_Opacity/masks/Lung_Opacity-3702.png,True,False,299,299,L,121.44,e7273dd093661a61456c422c1e5b3162
Lung_Opacity-3703.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3703.png,Lung_Opacity/images/Lung_Opacity-3703.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3703.png,Lung_Opacity/masks/Lung_Opacity-3703.png,True,False,299,299,L,110.41,2a7dbce3dddd69c491dc78d4b0b12296
Lung_Opacity-3704.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3704.png,Lung_Opacity/images/Lung_Opacity-3704.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3704.png,Lung_Opacity/masks/Lung_Opacity-3704.png,True,False,299,299,L,114.02,52227e35bac7afba6bd6aa3df931c74c
Lung_Opacity-3705.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3705.png,Lung_Opacity/images/Lung_Opacity-3705.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3705.png,Lung_Opacity/masks/Lung_Opacity-3705.png,True,False,299,299,L,106.2,74064bdd3c3442d79cb0db87e325f418
Lung_Opacity-3706.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3706.png,Lung_Opacity/images/Lung_Opacity-3706.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3706.png,Lung_Opacity/masks/Lung_Opacity-3706.png,True,False,299,299,L,110.53,a0ce821b006ac12ef6611d48f0996196
Lung_Opacity-3707.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3707.png,Lung_Opacity/images/Lung_Opacity-3707.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3707.png,Lung_Opacity/masks/Lung_Opacity-3707.png,True,False,299,299,L,112.91,fef2f4c4d0505f89c33e9949c8405332
Lung_Opacity-3708.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3708.png,Lung_Opacity/images/Lung_Opacity-3708.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3708.png,Lung_Opacity/masks/Lung_Opacity-3708.png,True,False,299,299,L,159.25,cba597aa85e0e7d29c93256e09584e9f
Lung_Opacity-3709.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3709.png,Lung_Opacity/images/Lung_Opacity-3709.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3709.png,Lung_Opacity/masks/Lung_Opacity-3709.png,True,False,299,299,L,121.38,6a2bde0a01a360ddc95a1a676c29dbeb
Lung_Opacity-371.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-371.png,Lung_Opacity/images/Lung_Opacity-371.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-371.png,Lung_Opacity/masks/Lung_Opacity-371.png,True,False,299,299,L,65.16,a9a0514dd8e185c2bfb4f4402dc77a1b
Lung_Opacity-3710.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3710.png,Lung_Opacity/images/Lung_Opacity-3710.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3710.png,Lung_Opacity/masks/Lung_Opacity-3710.png,True,False,299,299,L,108.57,a046f820884428d7e3c1a8545b602491
Lung_Opacity-3711.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3711.png,Lung_Opacity/images/Lung_Opacity-3711.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3711.png,Lung_Opacity/masks/Lung_Opacity-3711.png,True,False,299,299,L,101.98,f336242b16f55ca855503bcd4ece1050
Lung_Opacity-3712.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3712.png,Lung_Opacity/images/Lung_Opacity-3712.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3712.png,Lung_Opacity/masks/Lung_Opacity-3712.png,True,False,299,299,L,115.59,1278aa250e20df753d36535964db7b5d
Lung_Opacity-3713.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3713.png,Lung_Opacity/images/Lung_Opacity-3713.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3713.png,Lung_Opacity/masks/Lung_Opacity-3713.png,True,False,299,299,L,98.8,9522911b8ddec9e4e869a9ddb31d7310
Lung_Opacity-3714.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3714.png,Lung_Opacity/images/Lung_Opacity-3714.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3714.png,Lung_Opacity/masks/Lung_Opacity-3714.png,True,False,299,299,L,144.04,78a2f831a67d58d871ef54224da665b3
Lung_Opacity-3715.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3715.png,Lung_Opacity/images/Lung_Opacity-3715.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3715.png,Lung_Opacity/masks/Lung_Opacity-3715.png,True,False,299,299,L,111.29,b4fc48351d777e22541eac72d46996dd
Lung_Opacity-3716.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3716.png,Lung_Opacity/images/Lung_Opacity-3716.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3716.png,Lung_Opacity/masks/Lung_Opacity-3716.png,True,False,299,299,L,113.4,8a2d8da66c0dc66e2c11d3df99bb596a
Lung_Opacity-3717.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3717.png,Lung_Opacity/images/Lung_Opacity-3717.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3717.png,Lung_Opacity/masks/Lung_Opacity-3717.png,True,False,299,299,L,112.95,4b8105f296bcdcbdca410d3453822001
Lung_Opacity-3718.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3718.png,Lung_Opacity/images/Lung_Opacity-3718.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3718.png,Lung_Opacity/masks/Lung_Opacity-3718.png,True,False,299,299,L,107.2,e1e44512bd4996602567fdec716d2cae
Lung_Opacity-3719.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3719.png,Lung_Opacity/images/Lung_Opacity-3719.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3719.png,Lung_Opacity/masks/Lung_Opacity-3719.png,True,False,299,299,L,109.51,33571462e29022decf479fcc7d17fa0c
Lung_Opacity-372.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-372.png,Lung_Opacity/images/Lung_Opacity-372.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-372.png,Lung_Opacity/masks/Lung_Opacity-372.png,True,False,299,299,L,133.33,406beb1f25bdaea6410a1e7a2e1761cc
Lung_Opacity-3720.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3720.png,Lung_Opacity/images/Lung_Opacity-3720.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3720.png,Lung_Opacity/masks/Lung_Opacity-3720.png,True,False,299,299,L,127.27,6385ad20f394be172c09d8ab819c18df
Lung_Opacity-3721.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3721.png,Lung_Opacity/images/Lung_Opacity-3721.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3721.png,Lung_Opacity/masks/Lung_Opacity-3721.png,True,False,299,299,L,113.07,0cd2d6a238a77fe304d1cb8c155fbf63
Lung_Opacity-3722.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3722.png,Lung_Opacity/images/Lung_Opacity-3722.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3722.png,Lung_Opacity/masks/Lung_Opacity-3722.png,True,False,299,299,L,111.96,f268aa85243dc22499493edb7aeafbf2
Lung_Opacity-3723.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3723.png,Lung_Opacity/images/Lung_Opacity-3723.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3723.png,Lung_Opacity/masks/Lung_Opacity-3723.png,True,False,299,299,L,132.68,853e84f1b643b2000befd7b170807da0
Lung_Opacity-3724.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3724.png,Lung_Opacity/images/Lung_Opacity-3724.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3724.png,Lung_Opacity/masks/Lung_Opacity-3724.png,True,False,299,299,L,115.5,cf1a2051c1cf7ca86ef51e1c71ae1c43
Lung_Opacity-3725.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3725.png,Lung_Opacity/images/Lung_Opacity-3725.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3725.png,Lung_Opacity/masks/Lung_Opacity-3725.png,True,False,299,299,L,88.97,de2764d3176497e1bf8cb2d57ac3aadd
Lung_Opacity-3726.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3726.png,Lung_Opacity/images/Lung_Opacity-3726.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3726.png,Lung_Opacity/masks/Lung_Opacity-3726.png,True,False,299,299,L,131.7,3ba70a9bc52af0e1b4cc7fca93e0ca6e
Lung_Opacity-3727.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3727.png,Lung_Opacity/images/Lung_Opacity-3727.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3727.png,Lung_Opacity/masks/Lung_Opacity-3727.png,True,False,299,299,L,170.7,e118c6abe840236d4f3aedcc1e5df475
Lung_Opacity-3728.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3728.png,Lung_Opacity/images/Lung_Opacity-3728.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3728.png,Lung_Opacity/masks/Lung_Opacity-3728.png,True,False,299,299,L,115.08,7560755a91897656d81dc41c66146626
Lung_Opacity-3729.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3729.png,Lung_Opacity/images/Lung_Opacity-3729.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3729.png,Lung_Opacity/masks/Lung_Opacity-3729.png,True,False,299,299,L,113.51,e4aafb49dc1ec6db288b347824c7c17c
Lung_Opacity-373.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-373.png,Lung_Opacity/images/Lung_Opacity-373.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-373.png,Lung_Opacity/masks/Lung_Opacity-373.png,True,False,299,299,L,121.38,673167ea089a8be3df3c70bd76b1aeae
Lung_Opacity-3730.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3730.png,Lung_Opacity/images/Lung_Opacity-3730.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3730.png,Lung_Opacity/masks/Lung_Opacity-3730.png,True,False,299,299,L,115.71,b3400d9803bec4b1fe8038f784de5949
Lung_Opacity-3731.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3731.png,Lung_Opacity/images/Lung_Opacity-3731.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3731.png,Lung_Opacity/masks/Lung_Opacity-3731.png,True,False,299,299,L,111.12,6deda2c1e28f7e28d1de50b4c59c23c0
Lung_Opacity-3732.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3732.png,Lung_Opacity/images/Lung_Opacity-3732.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3732.png,Lung_Opacity/masks/Lung_Opacity-3732.png,True,False,299,299,L,111.82,c207f0c0fc8c3190ca79ac51b70c0bf5
Lung_Opacity-3733.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3733.png,Lung_Opacity/images/Lung_Opacity-3733.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3733.png,Lung_Opacity/masks/Lung_Opacity-3733.png,True,False,299,299,L,161.41,fd3b07790a69444c19102e863d5441cb
Lung_Opacity-3734.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3734.png,Lung_Opacity/images/Lung_Opacity-3734.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3734.png,Lung_Opacity/masks/Lung_Opacity-3734.png,True,False,299,299,L,134.61,451e235b1d297411e4fa4447b86e3466
Lung_Opacity-3735.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3735.png,Lung_Opacity/images/Lung_Opacity-3735.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3735.png,Lung_Opacity/masks/Lung_Opacity-3735.png,True,False,299,299,L,162.97,bacba5966bb63f8a6f117d9d52c18daa
Lung_Opacity-3736.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3736.png,Lung_Opacity/images/Lung_Opacity-3736.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3736.png,Lung_Opacity/masks/Lung_Opacity-3736.png,True,False,299,299,L,148.91,fa0a18812482980b12cd7f38f0f7b932
Lung_Opacity-3737.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3737.png,Lung_Opacity/images/Lung_Opacity-3737.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3737.png,Lung_Opacity/masks/Lung_Opacity-3737.png,True,False,299,299,L,121.28,71fa1b008d41c7386dcfc2722c71bd20
Lung_Opacity-3738.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3738.png,Lung_Opacity/images/Lung_Opacity-3738.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3738.png,Lung_Opacity/masks/Lung_Opacity-3738.png,True,False,299,299,L,129.05,f50ece60fc42b7b2f997077f6573b3d2
Lung_Opacity-3739.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3739.png,Lung_Opacity/images/Lung_Opacity-3739.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3739.png,Lung_Opacity/masks/Lung_Opacity-3739.png,True,False,299,299,L,128.28,4e61c8a10cfb402f023244da321adf6f
Lung_Opacity-374.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-374.png,Lung_Opacity/images/Lung_Opacity-374.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-374.png,Lung_Opacity/masks/Lung_Opacity-374.png,True,False,299,299,L,83.98,ebcbfe3467df98ec39e9461ba08b9dee
Lung_Opacity-3740.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3740.png,Lung_Opacity/images/Lung_Opacity-3740.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3740.png,Lung_Opacity/masks/Lung_Opacity-3740.png,True,False,299,299,L,134.93,796683c18b8bb88c98f5ff4923b64249
Lung_Opacity-3741.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3741.png,Lung_Opacity/images/Lung_Opacity-3741.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3741.png,Lung_Opacity/masks/Lung_Opacity-3741.png,True,False,299,299,L,130.93,43894aefffcdd9134e76bd0c4a20bc12
Lung_Opacity-3742.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3742.png,Lung_Opacity/images/Lung_Opacity-3742.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3742.png,Lung_Opacity/masks/Lung_Opacity-3742.png,True,False,299,299,L,116.25,eeb7b4f72bc6a3157597cac30e810b7f
Lung_Opacity-3743.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3743.png,Lung_Opacity/images/Lung_Opacity-3743.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3743.png,Lung_Opacity/masks/Lung_Opacity-3743.png,True,False,299,299,L,108.79,2babdd3eacaa296121d18286cfac3af2
Lung_Opacity-3744.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3744.png,Lung_Opacity/images/Lung_Opacity-3744.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3744.png,Lung_Opacity/masks/Lung_Opacity-3744.png,True,False,299,299,L,143.75,57863f955ed72a81d811e7d58047ec50
Lung_Opacity-3745.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3745.png,Lung_Opacity/images/Lung_Opacity-3745.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3745.png,Lung_Opacity/masks/Lung_Opacity-3745.png,True,False,299,299,L,122.93,cf9d2825f31c47b892315390dfa1ef04
Lung_Opacity-3746.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3746.png,Lung_Opacity/images/Lung_Opacity-3746.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3746.png,Lung_Opacity/masks/Lung_Opacity-3746.png,True,False,299,299,L,109.11,6206cc1977df151bebc81e0f7a7a1f24
Lung_Opacity-3747.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3747.png,Lung_Opacity/images/Lung_Opacity-3747.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3747.png,Lung_Opacity/masks/Lung_Opacity-3747.png,True,False,299,299,L,118.16,7f9493c1507cc2b9c2cab20bfc02777b
Lung_Opacity-3748.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3748.png,Lung_Opacity/images/Lung_Opacity-3748.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3748.png,Lung_Opacity/masks/Lung_Opacity-3748.png,True,False,299,299,L,120.34,28eea700bc1b86ff329a1a69c66875cc
Lung_Opacity-3749.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3749.png,Lung_Opacity/images/Lung_Opacity-3749.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3749.png,Lung_Opacity/masks/Lung_Opacity-3749.png,True,False,299,299,L,104.79,c39430439e764e4fb9f2364de3fd0c20
Lung_Opacity-375.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-375.png,Lung_Opacity/images/Lung_Opacity-375.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-375.png,Lung_Opacity/masks/Lung_Opacity-375.png,True,False,299,299,L,128.89,bd1e1c6c7bfe0becead6b1c1d1ce265c
Lung_Opacity-3750.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3750.png,Lung_Opacity/images/Lung_Opacity-3750.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3750.png,Lung_Opacity/masks/Lung_Opacity-3750.png,True,False,299,299,L,153.69,fc8c6dcb8e8a949037ae43397df4e6a9
Lung_Opacity-3751.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3751.png,Lung_Opacity/images/Lung_Opacity-3751.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3751.png,Lung_Opacity/masks/Lung_Opacity-3751.png,True,False,299,299,L,113.6,a907a6c1ce3c6aa41408a3e2f2ba164c
Lung_Opacity-3752.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3752.png,Lung_Opacity/images/Lung_Opacity-3752.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3752.png,Lung_Opacity/masks/Lung_Opacity-3752.png,True,False,299,299,L,151.93,a33906b18404975c80ab33111e5393ce
Lung_Opacity-3753.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3753.png,Lung_Opacity/images/Lung_Opacity-3753.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3753.png,Lung_Opacity/masks/Lung_Opacity-3753.png,True,False,299,299,L,124.91,f237992eb5f7211b3e7bb93381f67404
Lung_Opacity-3754.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3754.png,Lung_Opacity/images/Lung_Opacity-3754.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3754.png,Lung_Opacity/masks/Lung_Opacity-3754.png,True,False,299,299,L,114.56,a7ef3e51f825001e240a484dec3904b7
Lung_Opacity-3755.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3755.png,Lung_Opacity/images/Lung_Opacity-3755.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3755.png,Lung_Opacity/masks/Lung_Opacity-3755.png,True,False,299,299,L,140.33,01bc91de0e0e9e5204ea3a7909009c67
Lung_Opacity-3756.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3756.png,Lung_Opacity/images/Lung_Opacity-3756.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3756.png,Lung_Opacity/masks/Lung_Opacity-3756.png,True,False,299,299,L,112.08,35febf119718ae44d9cb153795775d06
Lung_Opacity-3757.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3757.png,Lung_Opacity/images/Lung_Opacity-3757.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3757.png,Lung_Opacity/masks/Lung_Opacity-3757.png,True,False,299,299,L,117.53,9a7e8b405a19eb91761e7c1718f00bc2
Lung_Opacity-3758.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3758.png,Lung_Opacity/images/Lung_Opacity-3758.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3758.png,Lung_Opacity/masks/Lung_Opacity-3758.png,True,False,299,299,L,115.26,4e0e76949d1219067b3d7c13bad4e68c
Lung_Opacity-3759.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3759.png,Lung_Opacity/images/Lung_Opacity-3759.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3759.png,Lung_Opacity/masks/Lung_Opacity-3759.png,True,False,299,299,L,112.89,f077cd3e17b2fc9fc19e4d20d3c845f5
Lung_Opacity-376.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-376.png,Lung_Opacity/images/Lung_Opacity-376.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-376.png,Lung_Opacity/masks/Lung_Opacity-376.png,True,False,299,299,L,105.75,b5e042dbaff71f0ea866b0335fe56b43
Lung_Opacity-3760.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3760.png,Lung_Opacity/images/Lung_Opacity-3760.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3760.png,Lung_Opacity/masks/Lung_Opacity-3760.png,True,False,299,299,L,134.13,f7a4cae2847e29e505764c80fee22d42
Lung_Opacity-3761.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3761.png,Lung_Opacity/images/Lung_Opacity-3761.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3761.png,Lung_Opacity/masks/Lung_Opacity-3761.png,True,False,299,299,L,114.98,ab11dd83199c2ad7c4ba5d3956440546
Lung_Opacity-3762.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3762.png,Lung_Opacity/images/Lung_Opacity-3762.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3762.png,Lung_Opacity/masks/Lung_Opacity-3762.png,True,False,299,299,L,157.18,ce189b07e6ab0e9da5c75cd5a50a94bc
Lung_Opacity-3763.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3763.png,Lung_Opacity/images/Lung_Opacity-3763.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3763.png,Lung_Opacity/masks/Lung_Opacity-3763.png,True,False,299,299,L,108.02,7ccfb26b90b33a2e207c1cb8013e7b9d
Lung_Opacity-3764.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3764.png,Lung_Opacity/images/Lung_Opacity-3764.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3764.png,Lung_Opacity/masks/Lung_Opacity-3764.png,True,False,299,299,L,128.84,d5ce200f2b873f2dbb1ddbbff550b188
Lung_Opacity-3765.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3765.png,Lung_Opacity/images/Lung_Opacity-3765.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3765.png,Lung_Opacity/masks/Lung_Opacity-3765.png,True,False,299,299,L,163.92,20ff79e0b46fbb368b72a6db56309f53
Lung_Opacity-3766.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3766.png,Lung_Opacity/images/Lung_Opacity-3766.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3766.png,Lung_Opacity/masks/Lung_Opacity-3766.png,True,False,299,299,L,115.2,df30a277e921b12ffce89bb5bde51369
Lung_Opacity-3767.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3767.png,Lung_Opacity/images/Lung_Opacity-3767.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3767.png,Lung_Opacity/masks/Lung_Opacity-3767.png,True,False,299,299,L,154.04,e3940da1450a13dbd9055b1ef050ffe0
Lung_Opacity-3768.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3768.png,Lung_Opacity/images/Lung_Opacity-3768.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3768.png,Lung_Opacity/masks/Lung_Opacity-3768.png,True,False,299,299,L,105.1,032d64380f2419d46ec8cc4e8a53dff8
Lung_Opacity-3769.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3769.png,Lung_Opacity/images/Lung_Opacity-3769.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3769.png,Lung_Opacity/masks/Lung_Opacity-3769.png,True,False,299,299,L,170.04,30781ecc78e8cb6a971410e22697d20e
Lung_Opacity-377.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-377.png,Lung_Opacity/images/Lung_Opacity-377.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-377.png,Lung_Opacity/masks/Lung_Opacity-377.png,True,False,299,299,L,132.62,00ca7fc1451001eab5e780870994f829
Lung_Opacity-3770.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3770.png,Lung_Opacity/images/Lung_Opacity-3770.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3770.png,Lung_Opacity/masks/Lung_Opacity-3770.png,True,False,299,299,L,112.62,fa037a4a6ebfbda4704338dbb03da680
Lung_Opacity-3771.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3771.png,Lung_Opacity/images/Lung_Opacity-3771.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3771.png,Lung_Opacity/masks/Lung_Opacity-3771.png,True,False,299,299,L,106.28,5082ddf9787253c442bb9101b8f30791
Lung_Opacity-3772.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3772.png,Lung_Opacity/images/Lung_Opacity-3772.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3772.png,Lung_Opacity/masks/Lung_Opacity-3772.png,True,False,299,299,L,112.33,7e1588243eeef28d01c9265077f44084
Lung_Opacity-3773.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3773.png,Lung_Opacity/images/Lung_Opacity-3773.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3773.png,Lung_Opacity/masks/Lung_Opacity-3773.png,True,False,299,299,L,95.23,360fc79969d39b9f9d67a8f44927060b
Lung_Opacity-3774.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3774.png,Lung_Opacity/images/Lung_Opacity-3774.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3774.png,Lung_Opacity/masks/Lung_Opacity-3774.png,True,False,299,299,L,118.3,9662e37546ed26b829a9d83f9a94b910
Lung_Opacity-3775.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3775.png,Lung_Opacity/images/Lung_Opacity-3775.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3775.png,Lung_Opacity/masks/Lung_Opacity-3775.png,True,False,299,299,L,150.73,625902340f6af42a0bd03ef4f1132c82
Lung_Opacity-3776.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3776.png,Lung_Opacity/images/Lung_Opacity-3776.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3776.png,Lung_Opacity/masks/Lung_Opacity-3776.png,True,False,299,299,L,88.79,146af4e3e9d594ff9919aeaaef6b8159
Lung_Opacity-3777.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3777.png,Lung_Opacity/images/Lung_Opacity-3777.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3777.png,Lung_Opacity/masks/Lung_Opacity-3777.png,True,False,299,299,L,126.9,28022b125d2158f8f0ba05258d23d571
Lung_Opacity-3778.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3778.png,Lung_Opacity/images/Lung_Opacity-3778.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3778.png,Lung_Opacity/masks/Lung_Opacity-3778.png,True,False,299,299,L,149.05,e4dfcf4ef9eeca555f01e37ae3a89380
Lung_Opacity-3779.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3779.png,Lung_Opacity/images/Lung_Opacity-3779.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3779.png,Lung_Opacity/masks/Lung_Opacity-3779.png,True,False,299,299,L,166.3,fb0ea7a55ec7a8f4701a3a24d07c7faf
Lung_Opacity-378.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-378.png,Lung_Opacity/images/Lung_Opacity-378.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-378.png,Lung_Opacity/masks/Lung_Opacity-378.png,True,False,299,299,L,162.19,241beb8a44b5cf8a929718f9a9418ffe
Lung_Opacity-3780.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3780.png,Lung_Opacity/images/Lung_Opacity-3780.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3780.png,Lung_Opacity/masks/Lung_Opacity-3780.png,True,False,299,299,L,103.0,b89b39f3bd3d54f85360dfa62e65a80c
Lung_Opacity-3781.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3781.png,Lung_Opacity/images/Lung_Opacity-3781.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3781.png,Lung_Opacity/masks/Lung_Opacity-3781.png,True,False,299,299,L,151.58,39d5e5ea699a815c373c2e7a4ca91db1
Lung_Opacity-3782.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3782.png,Lung_Opacity/images/Lung_Opacity-3782.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3782.png,Lung_Opacity/masks/Lung_Opacity-3782.png,True,False,299,299,L,125.2,8b701336f28344771225877f28d67213
Lung_Opacity-3783.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3783.png,Lung_Opacity/images/Lung_Opacity-3783.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3783.png,Lung_Opacity/masks/Lung_Opacity-3783.png,True,False,299,299,L,111.13,3df309569d8158647d4d04d605c94e01
Lung_Opacity-3784.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3784.png,Lung_Opacity/images/Lung_Opacity-3784.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3784.png,Lung_Opacity/masks/Lung_Opacity-3784.png,True,False,299,299,L,183.17,dcf54891bc4a966a716c8161d43be7c0
Lung_Opacity-3785.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3785.png,Lung_Opacity/images/Lung_Opacity-3785.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3785.png,Lung_Opacity/masks/Lung_Opacity-3785.png,True,False,299,299,L,124.41,b1dc449c0660c4fe71df64fc1a53c91d
Lung_Opacity-3786.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3786.png,Lung_Opacity/images/Lung_Opacity-3786.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3786.png,Lung_Opacity/masks/Lung_Opacity-3786.png,True,False,299,299,L,165.43,7e49031963f2f2e0af67e739733eeb07
Lung_Opacity-3787.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3787.png,Lung_Opacity/images/Lung_Opacity-3787.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3787.png,Lung_Opacity/masks/Lung_Opacity-3787.png,True,False,299,299,L,103.31,96116d351ceb06264949bcc500e470eb
Lung_Opacity-3788.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3788.png,Lung_Opacity/images/Lung_Opacity-3788.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3788.png,Lung_Opacity/masks/Lung_Opacity-3788.png,True,False,299,299,L,75.85,a4d49d34660bef9e4f240415cddb9c37
Lung_Opacity-3789.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3789.png,Lung_Opacity/images/Lung_Opacity-3789.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3789.png,Lung_Opacity/masks/Lung_Opacity-3789.png,True,False,299,299,L,106.66,8c06caa89a34756e7dc12c8d7755c668
Lung_Opacity-379.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-379.png,Lung_Opacity/images/Lung_Opacity-379.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-379.png,Lung_Opacity/masks/Lung_Opacity-379.png,True,False,299,299,L,132.01,fc0ef8108743e132dfbcf43b2e2125b0
Lung_Opacity-3790.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3790.png,Lung_Opacity/images/Lung_Opacity-3790.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3790.png,Lung_Opacity/masks/Lung_Opacity-3790.png,True,False,299,299,L,127.33,c4e2f744107aeec5c4aa2bbf2f775a3a
Lung_Opacity-3791.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3791.png,Lung_Opacity/images/Lung_Opacity-3791.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3791.png,Lung_Opacity/masks/Lung_Opacity-3791.png,True,False,299,299,L,117.38,f5e47364d9bdde350d3af838a1ceba0f
Lung_Opacity-3792.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3792.png,Lung_Opacity/images/Lung_Opacity-3792.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3792.png,Lung_Opacity/masks/Lung_Opacity-3792.png,True,False,299,299,L,155.29,b9358a6b051e4a61c4db4074e19373f2
Lung_Opacity-3793.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3793.png,Lung_Opacity/images/Lung_Opacity-3793.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3793.png,Lung_Opacity/masks/Lung_Opacity-3793.png,True,False,299,299,L,96.49,1e7eec4bd44eb731a87e168423f79f3a
Lung_Opacity-3794.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3794.png,Lung_Opacity/images/Lung_Opacity-3794.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3794.png,Lung_Opacity/masks/Lung_Opacity-3794.png,True,False,299,299,L,135.28,0d1202dfb5399aa480e1e646fb49c90d
Lung_Opacity-3795.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3795.png,Lung_Opacity/images/Lung_Opacity-3795.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3795.png,Lung_Opacity/masks/Lung_Opacity-3795.png,True,False,299,299,L,160.2,5b595de26e353f202b0ffb90643a4c4c
Lung_Opacity-3796.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3796.png,Lung_Opacity/images/Lung_Opacity-3796.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3796.png,Lung_Opacity/masks/Lung_Opacity-3796.png,True,False,299,299,L,99.01,73297c6a7b1c3948dc2223084e1e3290
Lung_Opacity-3797.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3797.png,Lung_Opacity/images/Lung_Opacity-3797.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3797.png,Lung_Opacity/masks/Lung_Opacity-3797.png,True,False,299,299,L,115.7,3880f8a948c35663c0bdc7d0360fc466
Lung_Opacity-3798.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3798.png,Lung_Opacity/images/Lung_Opacity-3798.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3798.png,Lung_Opacity/masks/Lung_Opacity-3798.png,True,False,299,299,L,122.75,985354357687b560da7294f02d3f5a2c
Lung_Opacity-3799.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3799.png,Lung_Opacity/images/Lung_Opacity-3799.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3799.png,Lung_Opacity/masks/Lung_Opacity-3799.png,True,False,299,299,L,114.8,23bd41a438ce5562bc9ef7aff6ff6f7c
Lung_Opacity-38.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-38.png,Lung_Opacity/images/Lung_Opacity-38.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-38.png,Lung_Opacity/masks/Lung_Opacity-38.png,True,False,299,299,L,106.75,7fb2c367b90f5b21b3a3f5603ac50abc
Lung_Opacity-380.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-380.png,Lung_Opacity/images/Lung_Opacity-380.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-380.png,Lung_Opacity/masks/Lung_Opacity-380.png,True,False,299,299,L,64.81,059a56d13eb7e36a56d463568ee3aed7
Lung_Opacity-3800.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3800.png,Lung_Opacity/images/Lung_Opacity-3800.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3800.png,Lung_Opacity/masks/Lung_Opacity-3800.png,True,False,299,299,L,154.76,3a4092146a5bc51db2a37f827f85f6a1
Lung_Opacity-3801.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3801.png,Lung_Opacity/images/Lung_Opacity-3801.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3801.png,Lung_Opacity/masks/Lung_Opacity-3801.png,True,False,299,299,L,181.98,e2d96729e4a7a1636d9365d3efc93a95
Lung_Opacity-3802.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3802.png,Lung_Opacity/images/Lung_Opacity-3802.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3802.png,Lung_Opacity/masks/Lung_Opacity-3802.png,True,False,299,299,L,134.25,c9164a7fd6b9c552a7f147180b519694
Lung_Opacity-3803.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3803.png,Lung_Opacity/images/Lung_Opacity-3803.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3803.png,Lung_Opacity/masks/Lung_Opacity-3803.png,True,False,299,299,L,111.15,8639f4b6da000f606efbff453c3dc53a
Lung_Opacity-3804.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3804.png,Lung_Opacity/images/Lung_Opacity-3804.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3804.png,Lung_Opacity/masks/Lung_Opacity-3804.png,True,False,299,299,L,160.96,fa613ef73fc0963c54152f5be0becddc
Lung_Opacity-3805.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3805.png,Lung_Opacity/images/Lung_Opacity-3805.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3805.png,Lung_Opacity/masks/Lung_Opacity-3805.png,True,False,299,299,L,97.51,b4bd7bd8fc37926cd5e972e617fc1eac
Lung_Opacity-3806.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3806.png,Lung_Opacity/images/Lung_Opacity-3806.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3806.png,Lung_Opacity/masks/Lung_Opacity-3806.png,True,False,299,299,L,117.37,a3de72bbe6fdbf756a40b2c0f6abefc4
Lung_Opacity-3807.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3807.png,Lung_Opacity/images/Lung_Opacity-3807.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3807.png,Lung_Opacity/masks/Lung_Opacity-3807.png,True,False,299,299,L,137.72,d097b138f37720ea0c25aa11713d52a5
Lung_Opacity-3808.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3808.png,Lung_Opacity/images/Lung_Opacity-3808.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3808.png,Lung_Opacity/masks/Lung_Opacity-3808.png,True,False,299,299,L,111.53,1b3ad319ec4c4645812440f5020849a1
Lung_Opacity-3809.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3809.png,Lung_Opacity/images/Lung_Opacity-3809.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3809.png,Lung_Opacity/masks/Lung_Opacity-3809.png,True,False,299,299,L,172.28,314fd48acbcd67f64604b64f34aa8501
Lung_Opacity-381.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-381.png,Lung_Opacity/images/Lung_Opacity-381.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-381.png,Lung_Opacity/masks/Lung_Opacity-381.png,True,False,299,299,L,169.92,a2319f2911a577a26921d78fd640e93a
Lung_Opacity-3810.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3810.png,Lung_Opacity/images/Lung_Opacity-3810.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3810.png,Lung_Opacity/masks/Lung_Opacity-3810.png,True,False,299,299,L,116.02,457cd61ba5137507a141722fc2ab7f2b
Lung_Opacity-3811.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3811.png,Lung_Opacity/images/Lung_Opacity-3811.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3811.png,Lung_Opacity/masks/Lung_Opacity-3811.png,True,False,299,299,L,152.8,646a4bae40c9fb3b0cd20f0c3cb28121
Lung_Opacity-3812.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3812.png,Lung_Opacity/images/Lung_Opacity-3812.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3812.png,Lung_Opacity/masks/Lung_Opacity-3812.png,True,False,299,299,L,102.49,55087779cefccef70acab7e8bf83bccb
Lung_Opacity-3813.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3813.png,Lung_Opacity/images/Lung_Opacity-3813.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3813.png,Lung_Opacity/masks/Lung_Opacity-3813.png,True,False,299,299,L,105.15,a47e1778ca56dd94593874ab09f7799f
Lung_Opacity-3814.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3814.png,Lung_Opacity/images/Lung_Opacity-3814.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3814.png,Lung_Opacity/masks/Lung_Opacity-3814.png,True,False,299,299,L,156.84,a6085c1546d02e8a9dc552ae82ee2747
Lung_Opacity-3815.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3815.png,Lung_Opacity/images/Lung_Opacity-3815.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3815.png,Lung_Opacity/masks/Lung_Opacity-3815.png,True,False,299,299,L,110.86,b63504b39d11208f904a3124dcd1aef1
Lung_Opacity-3816.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3816.png,Lung_Opacity/images/Lung_Opacity-3816.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3816.png,Lung_Opacity/masks/Lung_Opacity-3816.png,True,False,299,299,L,133.87,7e8e8f24b0ce1938560bea3bd4b24f8b
Lung_Opacity-3817.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3817.png,Lung_Opacity/images/Lung_Opacity-3817.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3817.png,Lung_Opacity/masks/Lung_Opacity-3817.png,True,False,299,299,L,110.05,672235f5cbc0a0fdf8494f09e636aeca
Lung_Opacity-3818.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3818.png,Lung_Opacity/images/Lung_Opacity-3818.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3818.png,Lung_Opacity/masks/Lung_Opacity-3818.png,True,False,299,299,L,112.62,4fb5fae5c4a8fb08c6efa3103532c6b2
Lung_Opacity-3819.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3819.png,Lung_Opacity/images/Lung_Opacity-3819.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3819.png,Lung_Opacity/masks/Lung_Opacity-3819.png,True,False,299,299,L,116.45,3bacb3e6b170c4870446513ea793e052
Lung_Opacity-382.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-382.png,Lung_Opacity/images/Lung_Opacity-382.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-382.png,Lung_Opacity/masks/Lung_Opacity-382.png,True,False,299,299,L,164.99,1b81ef7b784280050660831a08e32d54
Lung_Opacity-3820.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3820.png,Lung_Opacity/images/Lung_Opacity-3820.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3820.png,Lung_Opacity/masks/Lung_Opacity-3820.png,True,False,299,299,L,150.66,91787b5a3a2f37d7bddfd4df597a4ef8
Lung_Opacity-3821.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3821.png,Lung_Opacity/images/Lung_Opacity-3821.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3821.png,Lung_Opacity/masks/Lung_Opacity-3821.png,True,False,299,299,L,59.44,a96c7d7185d81816a0c25094511058eb
Lung_Opacity-3822.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3822.png,Lung_Opacity/images/Lung_Opacity-3822.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3822.png,Lung_Opacity/masks/Lung_Opacity-3822.png,True,False,299,299,L,125.17,7abaa0b54396b2d41d5283fadb33a093
Lung_Opacity-3823.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3823.png,Lung_Opacity/images/Lung_Opacity-3823.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3823.png,Lung_Opacity/masks/Lung_Opacity-3823.png,True,False,299,299,L,165.19,6ae06200379ee1e733bc7e5a432d5367
Lung_Opacity-3824.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3824.png,Lung_Opacity/images/Lung_Opacity-3824.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3824.png,Lung_Opacity/masks/Lung_Opacity-3824.png,True,False,299,299,L,98.04,12351d0b512bdedad2720a317bf93597
Lung_Opacity-3825.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3825.png,Lung_Opacity/images/Lung_Opacity-3825.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3825.png,Lung_Opacity/masks/Lung_Opacity-3825.png,True,False,299,299,L,119.11,2a151e556f639d0dd97937837c5e09cb
Lung_Opacity-3826.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3826.png,Lung_Opacity/images/Lung_Opacity-3826.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3826.png,Lung_Opacity/masks/Lung_Opacity-3826.png,True,False,299,299,L,113.81,621031499f907b144c52463f2f79b67b
Lung_Opacity-3827.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3827.png,Lung_Opacity/images/Lung_Opacity-3827.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3827.png,Lung_Opacity/masks/Lung_Opacity-3827.png,True,False,299,299,L,154.3,a67531deb50dd7c9f9dfd05afed41883
Lung_Opacity-3828.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3828.png,Lung_Opacity/images/Lung_Opacity-3828.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3828.png,Lung_Opacity/masks/Lung_Opacity-3828.png,True,False,299,299,L,119.85,cf390745d5b8cde698664aefa68fd474
Lung_Opacity-3829.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3829.png,Lung_Opacity/images/Lung_Opacity-3829.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3829.png,Lung_Opacity/masks/Lung_Opacity-3829.png,True,False,299,299,L,134.65,7e204648c88eea1ac4fe4b7d55110c1d
Lung_Opacity-383.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-383.png,Lung_Opacity/images/Lung_Opacity-383.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-383.png,Lung_Opacity/masks/Lung_Opacity-383.png,True,False,299,299,L,96.85,7f39cf87fb01113bfce3abc1d13b6a41
Lung_Opacity-3830.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3830.png,Lung_Opacity/images/Lung_Opacity-3830.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3830.png,Lung_Opacity/masks/Lung_Opacity-3830.png,True,False,299,299,L,112.88,6a244068997c1a831a2f9a25d82e1534
Lung_Opacity-3831.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3831.png,Lung_Opacity/images/Lung_Opacity-3831.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3831.png,Lung_Opacity/masks/Lung_Opacity-3831.png,True,False,299,299,L,112.82,669fdc47bec21b75861448795dbbc4b3
Lung_Opacity-3832.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3832.png,Lung_Opacity/images/Lung_Opacity-3832.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3832.png,Lung_Opacity/masks/Lung_Opacity-3832.png,True,False,299,299,L,113.37,28f5bc41fb64a92bdd51bc1a53aa5c99
Lung_Opacity-3833.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3833.png,Lung_Opacity/images/Lung_Opacity-3833.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3833.png,Lung_Opacity/masks/Lung_Opacity-3833.png,True,False,299,299,L,169.57,bb260e050108d7a18c4e5cd76f2a3510
Lung_Opacity-3834.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3834.png,Lung_Opacity/images/Lung_Opacity-3834.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3834.png,Lung_Opacity/masks/Lung_Opacity-3834.png,True,False,299,299,L,164.14,1712c3ff45a3f5494ab0afe64831528d
Lung_Opacity-3835.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3835.png,Lung_Opacity/images/Lung_Opacity-3835.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3835.png,Lung_Opacity/masks/Lung_Opacity-3835.png,True,False,299,299,L,70.83,c065ec883c4164072689fe6d526040c5
Lung_Opacity-3836.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3836.png,Lung_Opacity/images/Lung_Opacity-3836.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3836.png,Lung_Opacity/masks/Lung_Opacity-3836.png,True,False,299,299,L,128.19,4cda723c2213f733bb115cf91bc56c8e
Lung_Opacity-3837.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3837.png,Lung_Opacity/images/Lung_Opacity-3837.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3837.png,Lung_Opacity/masks/Lung_Opacity-3837.png,True,False,299,299,L,137.81,216645f5d1d28076d362735708a6b6e4
Lung_Opacity-3838.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3838.png,Lung_Opacity/images/Lung_Opacity-3838.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3838.png,Lung_Opacity/masks/Lung_Opacity-3838.png,True,False,299,299,L,95.17,6e2575e5175b513f8076b12c090fb785
Lung_Opacity-3839.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3839.png,Lung_Opacity/images/Lung_Opacity-3839.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3839.png,Lung_Opacity/masks/Lung_Opacity-3839.png,True,False,299,299,L,119.35,50dbbdde8ab9960aadc653fd2776dafa
Lung_Opacity-384.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-384.png,Lung_Opacity/images/Lung_Opacity-384.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-384.png,Lung_Opacity/masks/Lung_Opacity-384.png,True,False,299,299,L,112.9,f9bb3ce224a52b19696c3c764cb2cc5d
Lung_Opacity-3840.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3840.png,Lung_Opacity/images/Lung_Opacity-3840.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3840.png,Lung_Opacity/masks/Lung_Opacity-3840.png,True,False,299,299,L,100.23,3fbb1358a04375b2a39eb66201258d07
Lung_Opacity-3841.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3841.png,Lung_Opacity/images/Lung_Opacity-3841.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3841.png,Lung_Opacity/masks/Lung_Opacity-3841.png,True,False,299,299,L,156.98,32188079d65cc60c4d400dfb271d31ce
Lung_Opacity-3842.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3842.png,Lung_Opacity/images/Lung_Opacity-3842.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3842.png,Lung_Opacity/masks/Lung_Opacity-3842.png,True,False,299,299,L,167.61,cf925981e0c1e5bab060afcc6a233706
Lung_Opacity-3843.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3843.png,Lung_Opacity/images/Lung_Opacity-3843.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3843.png,Lung_Opacity/masks/Lung_Opacity-3843.png,True,False,299,299,L,110.4,782e861fcbceb71432dee645de269974
Lung_Opacity-3844.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3844.png,Lung_Opacity/images/Lung_Opacity-3844.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3844.png,Lung_Opacity/masks/Lung_Opacity-3844.png,True,False,299,299,L,115.2,7d8a804d9762914558a17f219df91ceb
Lung_Opacity-3845.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3845.png,Lung_Opacity/images/Lung_Opacity-3845.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3845.png,Lung_Opacity/masks/Lung_Opacity-3845.png,True,False,299,299,L,122.08,f5427a9ec331dbbfc06772f1238b77c3
Lung_Opacity-3846.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3846.png,Lung_Opacity/images/Lung_Opacity-3846.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3846.png,Lung_Opacity/masks/Lung_Opacity-3846.png,True,False,299,299,L,159.3,83b735cc9f922bf7421f64eab05bcbd4
Lung_Opacity-3847.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3847.png,Lung_Opacity/images/Lung_Opacity-3847.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3847.png,Lung_Opacity/masks/Lung_Opacity-3847.png,True,False,299,299,L,119.29,8e710e0c0adce0210a2851254222a8c4
Lung_Opacity-3848.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3848.png,Lung_Opacity/images/Lung_Opacity-3848.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3848.png,Lung_Opacity/masks/Lung_Opacity-3848.png,True,False,299,299,L,174.6,b3d77f50e6911948553be77ecfdc8f79
Lung_Opacity-3849.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3849.png,Lung_Opacity/images/Lung_Opacity-3849.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3849.png,Lung_Opacity/masks/Lung_Opacity-3849.png,True,False,299,299,L,162.07,8c10321a0f95f75c1b676cf5fc3ac692
Lung_Opacity-385.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-385.png,Lung_Opacity/images/Lung_Opacity-385.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-385.png,Lung_Opacity/masks/Lung_Opacity-385.png,True,False,299,299,L,163.41,dc61cbf6407841441a373ea28f3956c5
Lung_Opacity-3850.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3850.png,Lung_Opacity/images/Lung_Opacity-3850.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3850.png,Lung_Opacity/masks/Lung_Opacity-3850.png,True,False,299,299,L,140.83,c505e5b8e3b7fcbfbdbf7e3698a6fa40
Lung_Opacity-3851.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3851.png,Lung_Opacity/images/Lung_Opacity-3851.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3851.png,Lung_Opacity/masks/Lung_Opacity-3851.png,True,False,299,299,L,125.31,5f614a59088bcd32ea5f09e020fcb5d2
Lung_Opacity-3852.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3852.png,Lung_Opacity/images/Lung_Opacity-3852.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3852.png,Lung_Opacity/masks/Lung_Opacity-3852.png,True,False,299,299,L,120.24,47aec7729b41e785330997757683a85c
Lung_Opacity-3853.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3853.png,Lung_Opacity/images/Lung_Opacity-3853.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3853.png,Lung_Opacity/masks/Lung_Opacity-3853.png,True,False,299,299,L,131.32,8cf0aa5032ae02e1f521e11f81639dbd
Lung_Opacity-3854.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3854.png,Lung_Opacity/images/Lung_Opacity-3854.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3854.png,Lung_Opacity/masks/Lung_Opacity-3854.png,True,False,299,299,L,151.5,fe84f1275e2278aaeeb594ec65ebdc99
Lung_Opacity-3855.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3855.png,Lung_Opacity/images/Lung_Opacity-3855.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3855.png,Lung_Opacity/masks/Lung_Opacity-3855.png,True,False,299,299,L,136.78,9182e5e6c8ccb71246e86b0af49822fb
Lung_Opacity-3856.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3856.png,Lung_Opacity/images/Lung_Opacity-3856.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3856.png,Lung_Opacity/masks/Lung_Opacity-3856.png,True,False,299,299,L,127.93,2cfeb108e5bcd14fb47926cefe79d0f3
Lung_Opacity-3857.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3857.png,Lung_Opacity/images/Lung_Opacity-3857.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3857.png,Lung_Opacity/masks/Lung_Opacity-3857.png,True,False,299,299,L,147.51,0478bfd45e57148600054f75d935f9c7
Lung_Opacity-3858.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3858.png,Lung_Opacity/images/Lung_Opacity-3858.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3858.png,Lung_Opacity/masks/Lung_Opacity-3858.png,True,False,299,299,L,138.21,3899c20769fa7d96c2b7f4d86f6e1f69
Lung_Opacity-3859.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3859.png,Lung_Opacity/images/Lung_Opacity-3859.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3859.png,Lung_Opacity/masks/Lung_Opacity-3859.png,True,False,299,299,L,153.52,9a8598cc59db8913a003518af1d249cc
Lung_Opacity-386.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-386.png,Lung_Opacity/images/Lung_Opacity-386.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-386.png,Lung_Opacity/masks/Lung_Opacity-386.png,True,False,299,299,L,116.98,7a1b3af29058a5fa1f6cbca742af06b7
Lung_Opacity-3860.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3860.png,Lung_Opacity/images/Lung_Opacity-3860.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3860.png,Lung_Opacity/masks/Lung_Opacity-3860.png,True,False,299,299,L,161.6,b8134daf62f52422e09046d73f5de382
Lung_Opacity-3861.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3861.png,Lung_Opacity/images/Lung_Opacity-3861.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3861.png,Lung_Opacity/masks/Lung_Opacity-3861.png,True,False,299,299,L,151.16,8d64432fb81dc3a5d3f73563b4267da8
Lung_Opacity-3862.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3862.png,Lung_Opacity/images/Lung_Opacity-3862.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3862.png,Lung_Opacity/masks/Lung_Opacity-3862.png,True,False,299,299,L,109.69,0cf5123f61cf8098b9a8244b9c8d9141
Lung_Opacity-3863.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3863.png,Lung_Opacity/images/Lung_Opacity-3863.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3863.png,Lung_Opacity/masks/Lung_Opacity-3863.png,True,False,299,299,L,127.27,87b262b4617778e6ec599ee638aba9bd
Lung_Opacity-3864.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3864.png,Lung_Opacity/images/Lung_Opacity-3864.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3864.png,Lung_Opacity/masks/Lung_Opacity-3864.png,True,False,299,299,L,129.33,c3843919fc4ffa35219d377e02e3c511
Lung_Opacity-3865.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3865.png,Lung_Opacity/images/Lung_Opacity-3865.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3865.png,Lung_Opacity/masks/Lung_Opacity-3865.png,True,False,299,299,L,151.83,f29a58a4f02a9affca07e90376fd6540
Lung_Opacity-3866.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3866.png,Lung_Opacity/images/Lung_Opacity-3866.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3866.png,Lung_Opacity/masks/Lung_Opacity-3866.png,True,False,299,299,L,104.34,5029601236ed24d5982b067d1c818263
Lung_Opacity-3867.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3867.png,Lung_Opacity/images/Lung_Opacity-3867.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3867.png,Lung_Opacity/masks/Lung_Opacity-3867.png,True,False,299,299,L,117.37,8a20f3374929bf7a54e2588d70b5f403
Lung_Opacity-3868.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3868.png,Lung_Opacity/images/Lung_Opacity-3868.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3868.png,Lung_Opacity/masks/Lung_Opacity-3868.png,True,False,299,299,L,117.74,2fd67f6d0424b915b60a5f46279e40c1
Lung_Opacity-3869.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3869.png,Lung_Opacity/images/Lung_Opacity-3869.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3869.png,Lung_Opacity/masks/Lung_Opacity-3869.png,True,False,299,299,L,161.9,0d8e31ee9f311f29956ca0e1e8c198c6
Lung_Opacity-387.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-387.png,Lung_Opacity/images/Lung_Opacity-387.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-387.png,Lung_Opacity/masks/Lung_Opacity-387.png,True,False,299,299,L,97.85,026300abee7294cadd74408abc5052e1
Lung_Opacity-3870.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3870.png,Lung_Opacity/images/Lung_Opacity-3870.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3870.png,Lung_Opacity/masks/Lung_Opacity-3870.png,True,False,299,299,L,109.87,1db19ae3e6739bb7c703bd46e4cf1148
Lung_Opacity-3871.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3871.png,Lung_Opacity/images/Lung_Opacity-3871.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3871.png,Lung_Opacity/masks/Lung_Opacity-3871.png,True,False,299,299,L,146.38,3ff50d4d21851c4f45eef049f8c2b03f
Lung_Opacity-3872.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3872.png,Lung_Opacity/images/Lung_Opacity-3872.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3872.png,Lung_Opacity/masks/Lung_Opacity-3872.png,True,False,299,299,L,113.09,d3c048e3d0389e9e0af21a3962b3d459
Lung_Opacity-3873.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3873.png,Lung_Opacity/images/Lung_Opacity-3873.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3873.png,Lung_Opacity/masks/Lung_Opacity-3873.png,True,False,299,299,L,121.79,d3cb0a6d1c3bdcb4e8f03f2d9f403ebb
Lung_Opacity-3874.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3874.png,Lung_Opacity/images/Lung_Opacity-3874.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3874.png,Lung_Opacity/masks/Lung_Opacity-3874.png,True,False,299,299,L,99.84,688cd66b03198fe086b0aa651cc3c2af
Lung_Opacity-3875.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3875.png,Lung_Opacity/images/Lung_Opacity-3875.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3875.png,Lung_Opacity/masks/Lung_Opacity-3875.png,True,False,299,299,L,112.2,68a4329eb26ca0916e603426239ce6b0
Lung_Opacity-3876.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3876.png,Lung_Opacity/images/Lung_Opacity-3876.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3876.png,Lung_Opacity/masks/Lung_Opacity-3876.png,True,False,299,299,L,73.54,83778a53679ec7e073a05f3eb231c6e7
Lung_Opacity-3877.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3877.png,Lung_Opacity/images/Lung_Opacity-3877.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3877.png,Lung_Opacity/masks/Lung_Opacity-3877.png,True,False,299,299,L,132.55,ab34d2818af1e25b21c1a12acb3cf21d
Lung_Opacity-3878.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3878.png,Lung_Opacity/images/Lung_Opacity-3878.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3878.png,Lung_Opacity/masks/Lung_Opacity-3878.png,True,False,299,299,L,131.43,06b4a240be9e991df0b4f3536d02890e
Lung_Opacity-3879.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3879.png,Lung_Opacity/images/Lung_Opacity-3879.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3879.png,Lung_Opacity/masks/Lung_Opacity-3879.png,True,False,299,299,L,135.59,b8e3ee799e812de8810c6051b8a68fc2
Lung_Opacity-388.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-388.png,Lung_Opacity/images/Lung_Opacity-388.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-388.png,Lung_Opacity/masks/Lung_Opacity-388.png,True,False,299,299,L,162.33,8439c1286bc05d424c52c55adbc3984b
Lung_Opacity-3880.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3880.png,Lung_Opacity/images/Lung_Opacity-3880.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3880.png,Lung_Opacity/masks/Lung_Opacity-3880.png,True,False,299,299,L,132.2,cff91acb0edf220998720b13e25545fd
Lung_Opacity-3881.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3881.png,Lung_Opacity/images/Lung_Opacity-3881.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3881.png,Lung_Opacity/masks/Lung_Opacity-3881.png,True,False,299,299,L,162.85,7b7353ebb7b0aac10de6891beeedad57
Lung_Opacity-3882.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3882.png,Lung_Opacity/images/Lung_Opacity-3882.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3882.png,Lung_Opacity/masks/Lung_Opacity-3882.png,True,False,299,299,L,113.31,c442474466d81f781706a7c982fc9050
Lung_Opacity-3883.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3883.png,Lung_Opacity/images/Lung_Opacity-3883.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3883.png,Lung_Opacity/masks/Lung_Opacity-3883.png,True,False,299,299,L,166.16,ace57d528edad7964269eaaad1a56f3d
Lung_Opacity-3884.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3884.png,Lung_Opacity/images/Lung_Opacity-3884.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3884.png,Lung_Opacity/masks/Lung_Opacity-3884.png,True,False,299,299,L,141.42,2d648518f69dfd5bd8d01acfab3c4ba6
Lung_Opacity-3885.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3885.png,Lung_Opacity/images/Lung_Opacity-3885.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3885.png,Lung_Opacity/masks/Lung_Opacity-3885.png,True,False,299,299,L,176.19,abe8ce095d558d2a09228f17b085c4a0
Lung_Opacity-3886.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3886.png,Lung_Opacity/images/Lung_Opacity-3886.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3886.png,Lung_Opacity/masks/Lung_Opacity-3886.png,True,False,299,299,L,99.45,5bcde3809d29c728773f44bc45ff4e67
Lung_Opacity-3887.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3887.png,Lung_Opacity/images/Lung_Opacity-3887.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3887.png,Lung_Opacity/masks/Lung_Opacity-3887.png,True,False,299,299,L,164.13,b8d345f36fb9895bc34f4e335e8fed9f
Lung_Opacity-3888.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3888.png,Lung_Opacity/images/Lung_Opacity-3888.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3888.png,Lung_Opacity/masks/Lung_Opacity-3888.png,True,False,299,299,L,125.22,19c601e2c50aed67682906834e5349b1
Lung_Opacity-3889.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3889.png,Lung_Opacity/images/Lung_Opacity-3889.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3889.png,Lung_Opacity/masks/Lung_Opacity-3889.png,True,False,299,299,L,103.32,103fc7d690caa19dd99f433c2f957a35
Lung_Opacity-389.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-389.png,Lung_Opacity/images/Lung_Opacity-389.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-389.png,Lung_Opacity/masks/Lung_Opacity-389.png,True,False,299,299,L,122.1,a93bc1683a001fc6a680fa9cc1a2550f
Lung_Opacity-3890.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3890.png,Lung_Opacity/images/Lung_Opacity-3890.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3890.png,Lung_Opacity/masks/Lung_Opacity-3890.png,True,False,299,299,L,116.84,369db5b93befa45ff54f4fa8442285e9
Lung_Opacity-3891.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3891.png,Lung_Opacity/images/Lung_Opacity-3891.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3891.png,Lung_Opacity/masks/Lung_Opacity-3891.png,True,False,299,299,L,127.67,e2f1f12e87f7891f5d97079da559a5e2
Lung_Opacity-3892.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3892.png,Lung_Opacity/images/Lung_Opacity-3892.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3892.png,Lung_Opacity/masks/Lung_Opacity-3892.png,True,False,299,299,L,110.61,ed52defc63efc8836f33b70b7cda85f9
Lung_Opacity-3893.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3893.png,Lung_Opacity/images/Lung_Opacity-3893.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3893.png,Lung_Opacity/masks/Lung_Opacity-3893.png,True,False,299,299,L,154.69,1a0098f51a9994dda4cdcfe8553ca5f2
Lung_Opacity-3894.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3894.png,Lung_Opacity/images/Lung_Opacity-3894.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3894.png,Lung_Opacity/masks/Lung_Opacity-3894.png,True,False,299,299,L,144.87,09bec86166a8bea71dffd356734478f1
Lung_Opacity-3895.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3895.png,Lung_Opacity/images/Lung_Opacity-3895.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3895.png,Lung_Opacity/masks/Lung_Opacity-3895.png,True,False,299,299,L,123.38,087834d8984cb31ef64e1f088910f46a
Lung_Opacity-3896.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3896.png,Lung_Opacity/images/Lung_Opacity-3896.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3896.png,Lung_Opacity/masks/Lung_Opacity-3896.png,True,False,299,299,L,143.01,dd98a11dbc01547bdb638753e094c391
Lung_Opacity-3897.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3897.png,Lung_Opacity/images/Lung_Opacity-3897.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3897.png,Lung_Opacity/masks/Lung_Opacity-3897.png,True,False,299,299,L,137.57,4baa54668dc8d39745a9a44b11bed3cf
Lung_Opacity-3898.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3898.png,Lung_Opacity/images/Lung_Opacity-3898.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3898.png,Lung_Opacity/masks/Lung_Opacity-3898.png,True,False,299,299,L,165.65,8a900842fcd7c3b401e2cae0d7a579e0
Lung_Opacity-3899.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3899.png,Lung_Opacity/images/Lung_Opacity-3899.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3899.png,Lung_Opacity/masks/Lung_Opacity-3899.png,True,False,299,299,L,144.19,594494fe33fed59dabe9a27eeed486a2
Lung_Opacity-39.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-39.png,Lung_Opacity/images/Lung_Opacity-39.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-39.png,Lung_Opacity/masks/Lung_Opacity-39.png,True,False,299,299,L,149.45,bdbc6cfc7f9256c6d9ed23a5c3b7b7a2
Lung_Opacity-390.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-390.png,Lung_Opacity/images/Lung_Opacity-390.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-390.png,Lung_Opacity/masks/Lung_Opacity-390.png,True,False,299,299,L,131.39,206447810bd72a9e4b5fc5d72dd86833
Lung_Opacity-3900.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3900.png,Lung_Opacity/images/Lung_Opacity-3900.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3900.png,Lung_Opacity/masks/Lung_Opacity-3900.png,True,False,299,299,L,167.42,9583f2c8ac4a2b37333d0759b0c7409d
Lung_Opacity-3901.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3901.png,Lung_Opacity/images/Lung_Opacity-3901.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3901.png,Lung_Opacity/masks/Lung_Opacity-3901.png,True,False,299,299,L,163.54,9692a96d2812ac8bec5493c9b11794cf
Lung_Opacity-3902.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3902.png,Lung_Opacity/images/Lung_Opacity-3902.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3902.png,Lung_Opacity/masks/Lung_Opacity-3902.png,True,False,299,299,L,122.09,64fd06a5314faa90ef0d2da53c700ef6
Lung_Opacity-3903.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3903.png,Lung_Opacity/images/Lung_Opacity-3903.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3903.png,Lung_Opacity/masks/Lung_Opacity-3903.png,True,False,299,299,L,103.22,fbea1753f563b46e0455f77033e3398e
Lung_Opacity-3904.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3904.png,Lung_Opacity/images/Lung_Opacity-3904.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3904.png,Lung_Opacity/masks/Lung_Opacity-3904.png,True,False,299,299,L,162.71,37afcf8f3cfc6f379ec1e15a1038789a
Lung_Opacity-3905.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3905.png,Lung_Opacity/images/Lung_Opacity-3905.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3905.png,Lung_Opacity/masks/Lung_Opacity-3905.png,True,False,299,299,L,135.06,96ef5ff2ad614eb4911c570ac5b279ba
Lung_Opacity-3906.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3906.png,Lung_Opacity/images/Lung_Opacity-3906.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3906.png,Lung_Opacity/masks/Lung_Opacity-3906.png,True,False,299,299,L,134.2,e34a2c30e6622c39e2628440b4eb506b
Lung_Opacity-3907.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3907.png,Lung_Opacity/images/Lung_Opacity-3907.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3907.png,Lung_Opacity/masks/Lung_Opacity-3907.png,True,False,299,299,L,117.56,63486a6f6393a987b5bc48173201dc4a
Lung_Opacity-3908.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3908.png,Lung_Opacity/images/Lung_Opacity-3908.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3908.png,Lung_Opacity/masks/Lung_Opacity-3908.png,True,False,299,299,L,106.89,3638dcb6ece1b0c4ab2548f9265511c6
Lung_Opacity-3909.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3909.png,Lung_Opacity/images/Lung_Opacity-3909.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3909.png,Lung_Opacity/masks/Lung_Opacity-3909.png,True,False,299,299,L,116.9,541721cbdad2a4c6a2c311a94bf457f9
Lung_Opacity-391.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-391.png,Lung_Opacity/images/Lung_Opacity-391.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-391.png,Lung_Opacity/masks/Lung_Opacity-391.png,True,False,299,299,L,119.31,c9229ac808eb86bcf7d131cb1f980e77
Lung_Opacity-3910.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3910.png,Lung_Opacity/images/Lung_Opacity-3910.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3910.png,Lung_Opacity/masks/Lung_Opacity-3910.png,True,False,299,299,L,175.9,a7141a7c7deccc6f7d88ceef6b16800f
Lung_Opacity-3911.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3911.png,Lung_Opacity/images/Lung_Opacity-3911.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3911.png,Lung_Opacity/masks/Lung_Opacity-3911.png,True,False,299,299,L,172.76,7383a011740e49d907f70c1fd8c4d75a
Lung_Opacity-3912.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3912.png,Lung_Opacity/images/Lung_Opacity-3912.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3912.png,Lung_Opacity/masks/Lung_Opacity-3912.png,True,False,299,299,L,108.29,ca9e48edf498181d6865c18c588ae8f1
Lung_Opacity-3913.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3913.png,Lung_Opacity/images/Lung_Opacity-3913.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3913.png,Lung_Opacity/masks/Lung_Opacity-3913.png,True,False,299,299,L,153.71,81b6b89da0466728bc14be4256bfa41c
Lung_Opacity-3914.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3914.png,Lung_Opacity/images/Lung_Opacity-3914.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3914.png,Lung_Opacity/masks/Lung_Opacity-3914.png,True,False,299,299,L,126.86,642abfc228186212d74dc444c3214a3e
Lung_Opacity-3915.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3915.png,Lung_Opacity/images/Lung_Opacity-3915.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3915.png,Lung_Opacity/masks/Lung_Opacity-3915.png,True,False,299,299,L,153.7,af66b3eb1043a2af6bda0e4ef97b59a6
Lung_Opacity-3916.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3916.png,Lung_Opacity/images/Lung_Opacity-3916.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3916.png,Lung_Opacity/masks/Lung_Opacity-3916.png,True,False,299,299,L,123.81,fd0c5f91a687f11fe70a5ed3475e3ff6
Lung_Opacity-3917.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3917.png,Lung_Opacity/images/Lung_Opacity-3917.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3917.png,Lung_Opacity/masks/Lung_Opacity-3917.png,True,False,299,299,L,112.8,62fd7fe09ba151a30abdb5632bebb948
Lung_Opacity-3918.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3918.png,Lung_Opacity/images/Lung_Opacity-3918.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3918.png,Lung_Opacity/masks/Lung_Opacity-3918.png,True,False,299,299,L,115.21,f8b1d8bf96dfc83954ad17f392459237
Lung_Opacity-3919.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3919.png,Lung_Opacity/images/Lung_Opacity-3919.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3919.png,Lung_Opacity/masks/Lung_Opacity-3919.png,True,False,299,299,L,114.92,006541cfa8244e937b58bcf7f467006b
Lung_Opacity-392.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-392.png,Lung_Opacity/images/Lung_Opacity-392.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-392.png,Lung_Opacity/masks/Lung_Opacity-392.png,True,False,299,299,L,140.66,b6bf49eba2aef58a4be0e7a7e93f28dc
Lung_Opacity-3920.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3920.png,Lung_Opacity/images/Lung_Opacity-3920.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3920.png,Lung_Opacity/masks/Lung_Opacity-3920.png,True,False,299,299,L,136.17,8f462de856d483a6d812871127f5159e
Lung_Opacity-3921.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3921.png,Lung_Opacity/images/Lung_Opacity-3921.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3921.png,Lung_Opacity/masks/Lung_Opacity-3921.png,True,False,299,299,L,148.81,21e6b28ed85f8b837447f2e68ee29a9e
Lung_Opacity-3922.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3922.png,Lung_Opacity/images/Lung_Opacity-3922.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3922.png,Lung_Opacity/masks/Lung_Opacity-3922.png,True,False,299,299,L,108.62,64e7c511f82565124d31a12a0fe3fa36
Lung_Opacity-3923.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3923.png,Lung_Opacity/images/Lung_Opacity-3923.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3923.png,Lung_Opacity/masks/Lung_Opacity-3923.png,True,False,299,299,L,149.18,d3896b1078f3a4d1c4dae995641ddfa8
Lung_Opacity-3924.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3924.png,Lung_Opacity/images/Lung_Opacity-3924.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3924.png,Lung_Opacity/masks/Lung_Opacity-3924.png,True,False,299,299,L,103.32,9704ac5c03381fd4200203efeab9e6c3
Lung_Opacity-3925.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3925.png,Lung_Opacity/images/Lung_Opacity-3925.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3925.png,Lung_Opacity/masks/Lung_Opacity-3925.png,True,False,299,299,L,132.74,6c35e1f52f67838baa574cfcb68a2d70
Lung_Opacity-3926.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3926.png,Lung_Opacity/images/Lung_Opacity-3926.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3926.png,Lung_Opacity/masks/Lung_Opacity-3926.png,True,False,299,299,L,126.36,3788af8a61f19a5900d832e44efcdcdf
Lung_Opacity-3927.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3927.png,Lung_Opacity/images/Lung_Opacity-3927.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3927.png,Lung_Opacity/masks/Lung_Opacity-3927.png,True,False,299,299,L,128.51,8d2ec2a3abb2562940a6a773e4747d47
Lung_Opacity-3928.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3928.png,Lung_Opacity/images/Lung_Opacity-3928.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3928.png,Lung_Opacity/masks/Lung_Opacity-3928.png,True,False,299,299,L,80.75,2520a1b49d5597ec6a810fdc8a0cc389
Lung_Opacity-3929.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3929.png,Lung_Opacity/images/Lung_Opacity-3929.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3929.png,Lung_Opacity/masks/Lung_Opacity-3929.png,True,False,299,299,L,97.57,03f5a5896da0f47c8c7b483b9d099268
Lung_Opacity-393.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-393.png,Lung_Opacity/images/Lung_Opacity-393.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-393.png,Lung_Opacity/masks/Lung_Opacity-393.png,True,False,299,299,L,156.12,dda63c5ab27c89bcee04885c5d05316d
Lung_Opacity-3930.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3930.png,Lung_Opacity/images/Lung_Opacity-3930.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3930.png,Lung_Opacity/masks/Lung_Opacity-3930.png,True,False,299,299,L,142.07,61e104fbc5984a7a6ed8d67a6c3526b6
Lung_Opacity-3931.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3931.png,Lung_Opacity/images/Lung_Opacity-3931.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3931.png,Lung_Opacity/masks/Lung_Opacity-3931.png,True,False,299,299,L,127.84,f985113f6fbed6f44b715b47da5ab640
Lung_Opacity-3932.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3932.png,Lung_Opacity/images/Lung_Opacity-3932.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3932.png,Lung_Opacity/masks/Lung_Opacity-3932.png,True,False,299,299,L,103.57,e5960cb0ae9a61945800cc3234312f2e
Lung_Opacity-3933.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3933.png,Lung_Opacity/images/Lung_Opacity-3933.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3933.png,Lung_Opacity/masks/Lung_Opacity-3933.png,True,False,299,299,L,116.3,41be9152bce35dda330bb2b00c88675f
Lung_Opacity-3934.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3934.png,Lung_Opacity/images/Lung_Opacity-3934.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3934.png,Lung_Opacity/masks/Lung_Opacity-3934.png,True,False,299,299,L,111.82,989d03402e73a80de1d05029d79ef759
Lung_Opacity-3935.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3935.png,Lung_Opacity/images/Lung_Opacity-3935.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3935.png,Lung_Opacity/masks/Lung_Opacity-3935.png,True,False,299,299,L,118.46,a9e0fb3b35f67a71fb00c2d57adafa15
Lung_Opacity-3936.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3936.png,Lung_Opacity/images/Lung_Opacity-3936.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3936.png,Lung_Opacity/masks/Lung_Opacity-3936.png,True,False,299,299,L,93.76,7d3c4cd4f03cbc2c89733be26e4900e9
Lung_Opacity-3937.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3937.png,Lung_Opacity/images/Lung_Opacity-3937.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3937.png,Lung_Opacity/masks/Lung_Opacity-3937.png,True,False,299,299,L,157.75,84c627775804255edeaf2504b156ac71
Lung_Opacity-3938.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3938.png,Lung_Opacity/images/Lung_Opacity-3938.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3938.png,Lung_Opacity/masks/Lung_Opacity-3938.png,True,False,299,299,L,115.18,7d822e86b4d7d84ec8fa20e58eb67ce3
Lung_Opacity-3939.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3939.png,Lung_Opacity/images/Lung_Opacity-3939.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3939.png,Lung_Opacity/masks/Lung_Opacity-3939.png,True,False,299,299,L,116.79,da7b13a6c4143415d4cbdbbe3c964e98
Lung_Opacity-394.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-394.png,Lung_Opacity/images/Lung_Opacity-394.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-394.png,Lung_Opacity/masks/Lung_Opacity-394.png,True,False,299,299,L,111.77,88a42239e9b7215695fb7f49f0d76a7d
Lung_Opacity-3940.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3940.png,Lung_Opacity/images/Lung_Opacity-3940.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3940.png,Lung_Opacity/masks/Lung_Opacity-3940.png,True,False,299,299,L,110.2,31b51be1f337f8c8085c64f05545ba6a
Lung_Opacity-3941.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3941.png,Lung_Opacity/images/Lung_Opacity-3941.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3941.png,Lung_Opacity/masks/Lung_Opacity-3941.png,True,False,299,299,L,110.06,ea83f479a5d76b847fdf0818d28f8201
Lung_Opacity-3942.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3942.png,Lung_Opacity/images/Lung_Opacity-3942.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3942.png,Lung_Opacity/masks/Lung_Opacity-3942.png,True,False,299,299,L,120.85,5e0ac1bce8956e855e1e746fc530d405
Lung_Opacity-3943.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3943.png,Lung_Opacity/images/Lung_Opacity-3943.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3943.png,Lung_Opacity/masks/Lung_Opacity-3943.png,True,False,299,299,L,115.9,bca299286b25579d8a487a82d1a2f65c
Lung_Opacity-3944.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3944.png,Lung_Opacity/images/Lung_Opacity-3944.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3944.png,Lung_Opacity/masks/Lung_Opacity-3944.png,True,False,299,299,L,129.84,e7ec3068bc639d375597e46856f68863
Lung_Opacity-3945.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3945.png,Lung_Opacity/images/Lung_Opacity-3945.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3945.png,Lung_Opacity/masks/Lung_Opacity-3945.png,True,False,299,299,L,117.84,ce9f0e2b5e7c6b578eb01dd0305b3876
Lung_Opacity-3946.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3946.png,Lung_Opacity/images/Lung_Opacity-3946.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3946.png,Lung_Opacity/masks/Lung_Opacity-3946.png,True,False,299,299,L,151.38,22b028a92cebfe73bddb9e5e2ddac16c
Lung_Opacity-3947.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3947.png,Lung_Opacity/images/Lung_Opacity-3947.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3947.png,Lung_Opacity/masks/Lung_Opacity-3947.png,True,False,299,299,L,115.75,aa3c4eb079ab57575bf2f612c112796b
Lung_Opacity-3948.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3948.png,Lung_Opacity/images/Lung_Opacity-3948.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3948.png,Lung_Opacity/masks/Lung_Opacity-3948.png,True,False,299,299,L,116.61,a31d3e37e2cd7e3a9723811fc2079d16
Lung_Opacity-3949.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3949.png,Lung_Opacity/images/Lung_Opacity-3949.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3949.png,Lung_Opacity/masks/Lung_Opacity-3949.png,True,False,299,299,L,137.45,c4df52112d46a6e026a041755676a467
Lung_Opacity-395.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-395.png,Lung_Opacity/images/Lung_Opacity-395.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-395.png,Lung_Opacity/masks/Lung_Opacity-395.png,True,False,299,299,L,175.58,29ac4764a3862390b076eacafada1963
Lung_Opacity-3950.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3950.png,Lung_Opacity/images/Lung_Opacity-3950.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3950.png,Lung_Opacity/masks/Lung_Opacity-3950.png,True,False,299,299,L,127.39,6a44400ce8a6f64d8de0a41f711ff158
Lung_Opacity-3951.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3951.png,Lung_Opacity/images/Lung_Opacity-3951.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3951.png,Lung_Opacity/masks/Lung_Opacity-3951.png,True,False,299,299,L,112.15,56b4807a395c82d87c8faeb56698067d
Lung_Opacity-3952.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3952.png,Lung_Opacity/images/Lung_Opacity-3952.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3952.png,Lung_Opacity/masks/Lung_Opacity-3952.png,True,False,299,299,L,110.58,639918730b9505f25a9972fe4f49efa7
Lung_Opacity-3953.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3953.png,Lung_Opacity/images/Lung_Opacity-3953.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3953.png,Lung_Opacity/masks/Lung_Opacity-3953.png,True,False,299,299,L,122.61,ce6da3ceb77ba6891d7ad7ebeb4d4cf9
Lung_Opacity-3954.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3954.png,Lung_Opacity/images/Lung_Opacity-3954.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3954.png,Lung_Opacity/masks/Lung_Opacity-3954.png,True,False,299,299,L,114.24,05ad5b62f550e3f5ef348f705aeadc37
Lung_Opacity-3955.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3955.png,Lung_Opacity/images/Lung_Opacity-3955.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3955.png,Lung_Opacity/masks/Lung_Opacity-3955.png,True,False,299,299,L,118.74,f3f968fb06085fa13eac25ef931e9839
Lung_Opacity-3956.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3956.png,Lung_Opacity/images/Lung_Opacity-3956.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3956.png,Lung_Opacity/masks/Lung_Opacity-3956.png,True,False,299,299,L,107.64,f3cdb81d1929ab6fafc09503c3a22536
Lung_Opacity-3957.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3957.png,Lung_Opacity/images/Lung_Opacity-3957.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3957.png,Lung_Opacity/masks/Lung_Opacity-3957.png,True,False,299,299,L,117.06,01c2bdda29d4415412586d22f898227f
Lung_Opacity-3958.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3958.png,Lung_Opacity/images/Lung_Opacity-3958.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3958.png,Lung_Opacity/masks/Lung_Opacity-3958.png,True,False,299,299,L,113.94,ccab77279fe979d65067c28a55956500
Lung_Opacity-3959.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3959.png,Lung_Opacity/images/Lung_Opacity-3959.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3959.png,Lung_Opacity/masks/Lung_Opacity-3959.png,True,False,299,299,L,141.09,f7e69c56698f73a4e56f34dcd9933548
Lung_Opacity-396.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-396.png,Lung_Opacity/images/Lung_Opacity-396.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-396.png,Lung_Opacity/masks/Lung_Opacity-396.png,True,False,299,299,L,106.63,15dba348d9071388363cfef895b8bae0
Lung_Opacity-3960.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3960.png,Lung_Opacity/images/Lung_Opacity-3960.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3960.png,Lung_Opacity/masks/Lung_Opacity-3960.png,True,False,299,299,L,158.26,0f77aba53a418cb93a2524df0403dd2b
Lung_Opacity-3961.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3961.png,Lung_Opacity/images/Lung_Opacity-3961.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3961.png,Lung_Opacity/masks/Lung_Opacity-3961.png,True,False,299,299,L,104.05,b5253abb3dd3fbd9886cc619b150f3be
Lung_Opacity-3962.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3962.png,Lung_Opacity/images/Lung_Opacity-3962.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3962.png,Lung_Opacity/masks/Lung_Opacity-3962.png,True,False,299,299,L,114.59,581e737ee870a5600024b19872e66545
Lung_Opacity-3963.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3963.png,Lung_Opacity/images/Lung_Opacity-3963.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3963.png,Lung_Opacity/masks/Lung_Opacity-3963.png,True,False,299,299,L,115.14,89d1fd48afda3ff85725a306c2cd6b44
Lung_Opacity-3964.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3964.png,Lung_Opacity/images/Lung_Opacity-3964.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3964.png,Lung_Opacity/masks/Lung_Opacity-3964.png,True,False,299,299,L,128.22,0e3d01fc5bb453f65f4978fad3894141
Lung_Opacity-3965.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3965.png,Lung_Opacity/images/Lung_Opacity-3965.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3965.png,Lung_Opacity/masks/Lung_Opacity-3965.png,True,False,299,299,L,96.85,5a11afba2d87c459a19138c8fb2e5bf6
Lung_Opacity-3966.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3966.png,Lung_Opacity/images/Lung_Opacity-3966.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3966.png,Lung_Opacity/masks/Lung_Opacity-3966.png,True,False,299,299,L,141.04,11b6ffb7f190d304aafa19b2b423411c
Lung_Opacity-3967.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3967.png,Lung_Opacity/images/Lung_Opacity-3967.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3967.png,Lung_Opacity/masks/Lung_Opacity-3967.png,True,False,299,299,L,154.2,aee92385736e804ad7d13ecb814e6c8a
Lung_Opacity-3968.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3968.png,Lung_Opacity/images/Lung_Opacity-3968.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3968.png,Lung_Opacity/masks/Lung_Opacity-3968.png,True,False,299,299,L,114.29,06a38423c662ebd38e44ec8ce749dd63
Lung_Opacity-3969.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3969.png,Lung_Opacity/images/Lung_Opacity-3969.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3969.png,Lung_Opacity/masks/Lung_Opacity-3969.png,True,False,299,299,L,103.39,e787a4c68d7238c43375719603d78946
Lung_Opacity-397.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-397.png,Lung_Opacity/images/Lung_Opacity-397.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-397.png,Lung_Opacity/masks/Lung_Opacity-397.png,True,False,299,299,L,117.08,da763395015a54d7f69f42fd166823d8
Lung_Opacity-3970.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3970.png,Lung_Opacity/images/Lung_Opacity-3970.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3970.png,Lung_Opacity/masks/Lung_Opacity-3970.png,True,False,299,299,L,124.41,955a51b56737a4b4a2e366530df428d0
Lung_Opacity-3971.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3971.png,Lung_Opacity/images/Lung_Opacity-3971.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3971.png,Lung_Opacity/masks/Lung_Opacity-3971.png,True,False,299,299,L,110.09,328a906362998aca7ee087afb92ae2a3
Lung_Opacity-3972.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3972.png,Lung_Opacity/images/Lung_Opacity-3972.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3972.png,Lung_Opacity/masks/Lung_Opacity-3972.png,True,False,299,299,L,116.98,b6f2a5cc4e308e524635a0959fa089f0
Lung_Opacity-3973.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3973.png,Lung_Opacity/images/Lung_Opacity-3973.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3973.png,Lung_Opacity/masks/Lung_Opacity-3973.png,True,False,299,299,L,106.58,50a50f50f9b68131599000987976922d
Lung_Opacity-3974.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3974.png,Lung_Opacity/images/Lung_Opacity-3974.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3974.png,Lung_Opacity/masks/Lung_Opacity-3974.png,True,False,299,299,L,105.84,54748c44ea46b9b035145926873a6707
Lung_Opacity-3975.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3975.png,Lung_Opacity/images/Lung_Opacity-3975.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3975.png,Lung_Opacity/masks/Lung_Opacity-3975.png,True,False,299,299,L,116.52,b6e607e474497f898fdf433eac718417
Lung_Opacity-3976.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3976.png,Lung_Opacity/images/Lung_Opacity-3976.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3976.png,Lung_Opacity/masks/Lung_Opacity-3976.png,True,False,299,299,L,146.2,9573e826da12122f7a65b89f60ab6482
Lung_Opacity-3977.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3977.png,Lung_Opacity/images/Lung_Opacity-3977.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3977.png,Lung_Opacity/masks/Lung_Opacity-3977.png,True,False,299,299,L,119.03,2bbe61dd2372a468ca77008cc20f5337
Lung_Opacity-3978.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3978.png,Lung_Opacity/images/Lung_Opacity-3978.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3978.png,Lung_Opacity/masks/Lung_Opacity-3978.png,True,False,299,299,L,114.36,0091abed4c0809d2be9bb7392cebdc60
Lung_Opacity-3979.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3979.png,Lung_Opacity/images/Lung_Opacity-3979.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3979.png,Lung_Opacity/masks/Lung_Opacity-3979.png,True,False,299,299,L,92.68,24040016718892d2cdf5f14f58d4ee86
Lung_Opacity-398.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-398.png,Lung_Opacity/images/Lung_Opacity-398.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-398.png,Lung_Opacity/masks/Lung_Opacity-398.png,True,False,299,299,L,118.12,52457e93522ae766e27832ebf3fdf1a3
Lung_Opacity-3980.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3980.png,Lung_Opacity/images/Lung_Opacity-3980.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3980.png,Lung_Opacity/masks/Lung_Opacity-3980.png,True,False,299,299,L,114.04,c54e63f7e315be5dfcea4ab1ffd960d3
Lung_Opacity-3981.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3981.png,Lung_Opacity/images/Lung_Opacity-3981.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3981.png,Lung_Opacity/masks/Lung_Opacity-3981.png,True,False,299,299,L,129.75,75a6761ea8df8a3521a890d80b02b383
Lung_Opacity-3982.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3982.png,Lung_Opacity/images/Lung_Opacity-3982.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3982.png,Lung_Opacity/masks/Lung_Opacity-3982.png,True,False,299,299,L,111.89,2f5df9b17c1f903238f2bb6a5e47a817
Lung_Opacity-3983.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3983.png,Lung_Opacity/images/Lung_Opacity-3983.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3983.png,Lung_Opacity/masks/Lung_Opacity-3983.png,True,False,299,299,L,107.36,9d8d600a65bfa59d6e4f2784fabb88ce
Lung_Opacity-3984.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3984.png,Lung_Opacity/images/Lung_Opacity-3984.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3984.png,Lung_Opacity/masks/Lung_Opacity-3984.png,True,False,299,299,L,126.46,b381bfd0f6838d06c76ba935977db762
Lung_Opacity-3985.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3985.png,Lung_Opacity/images/Lung_Opacity-3985.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3985.png,Lung_Opacity/masks/Lung_Opacity-3985.png,True,False,299,299,L,146.28,a7bab61c0a4a34be43ea4b719f528a97
Lung_Opacity-3986.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3986.png,Lung_Opacity/images/Lung_Opacity-3986.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3986.png,Lung_Opacity/masks/Lung_Opacity-3986.png,True,False,299,299,L,143.63,c4b322fb28ba426f385ed3a664987986
Lung_Opacity-3987.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3987.png,Lung_Opacity/images/Lung_Opacity-3987.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3987.png,Lung_Opacity/masks/Lung_Opacity-3987.png,True,False,299,299,L,96.82,8a2eb9e674e618428029cd76f01a3999
Lung_Opacity-3988.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3988.png,Lung_Opacity/images/Lung_Opacity-3988.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3988.png,Lung_Opacity/masks/Lung_Opacity-3988.png,True,False,299,299,L,125.02,8c74d3d7bfe2df65e79257d2f498419f
Lung_Opacity-3989.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3989.png,Lung_Opacity/images/Lung_Opacity-3989.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3989.png,Lung_Opacity/masks/Lung_Opacity-3989.png,True,False,299,299,L,145.94,16dea5eb181144aacd8bc03747c56ce6
Lung_Opacity-399.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-399.png,Lung_Opacity/images/Lung_Opacity-399.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-399.png,Lung_Opacity/masks/Lung_Opacity-399.png,True,False,299,299,L,99.13,d127bd4d1ec92f26c281e1aff45024c2
Lung_Opacity-3990.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3990.png,Lung_Opacity/images/Lung_Opacity-3990.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3990.png,Lung_Opacity/masks/Lung_Opacity-3990.png,True,False,299,299,L,125.48,fbe161174a5029d6cbaff47260451094
Lung_Opacity-3991.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3991.png,Lung_Opacity/images/Lung_Opacity-3991.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3991.png,Lung_Opacity/masks/Lung_Opacity-3991.png,True,False,299,299,L,113.53,11bbf0fb820c0cd39f9134a7139c88b1
Lung_Opacity-3992.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3992.png,Lung_Opacity/images/Lung_Opacity-3992.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3992.png,Lung_Opacity/masks/Lung_Opacity-3992.png,True,False,299,299,L,111.35,4779456f4f46b5e6830d01b72dc0bea4
Lung_Opacity-3993.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3993.png,Lung_Opacity/images/Lung_Opacity-3993.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3993.png,Lung_Opacity/masks/Lung_Opacity-3993.png,True,False,299,299,L,101.15,9f78666224f30349847c75884d423842
Lung_Opacity-3994.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3994.png,Lung_Opacity/images/Lung_Opacity-3994.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3994.png,Lung_Opacity/masks/Lung_Opacity-3994.png,True,False,299,299,L,106.3,d824286cacacbdff53f85136b1963039
Lung_Opacity-3995.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3995.png,Lung_Opacity/images/Lung_Opacity-3995.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3995.png,Lung_Opacity/masks/Lung_Opacity-3995.png,True,False,299,299,L,111.33,82b7175783731245551facb7b7883c18
Lung_Opacity-3996.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3996.png,Lung_Opacity/images/Lung_Opacity-3996.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3996.png,Lung_Opacity/masks/Lung_Opacity-3996.png,True,False,299,299,L,111.23,37881f41098748a96358fc6d9b577959
Lung_Opacity-3997.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3997.png,Lung_Opacity/images/Lung_Opacity-3997.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3997.png,Lung_Opacity/masks/Lung_Opacity-3997.png,True,False,299,299,L,130.33,682efa86bae1c0c0bffa09b23ce33406
Lung_Opacity-3998.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3998.png,Lung_Opacity/images/Lung_Opacity-3998.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3998.png,Lung_Opacity/masks/Lung_Opacity-3998.png,True,False,299,299,L,126.39,d85cc14291bf125d9ceb5cc9694c427d
Lung_Opacity-3999.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-3999.png,Lung_Opacity/images/Lung_Opacity-3999.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-3999.png,Lung_Opacity/masks/Lung_Opacity-3999.png,True,False,299,299,L,134.61,cbccd2c0f93e04ecc2cfc2c708aec6ea
Lung_Opacity-4.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4.png,Lung_Opacity/images/Lung_Opacity-4.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4.png,Lung_Opacity/masks/Lung_Opacity-4.png,True,False,299,299,L,133.14,9d608876a34c60f4555e78c77c815913
Lung_Opacity-40.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-40.png,Lung_Opacity/images/Lung_Opacity-40.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-40.png,Lung_Opacity/masks/Lung_Opacity-40.png,True,False,299,299,L,141.03,e08baef8dccbe996eb280bf4d9d0d7b7
Lung_Opacity-400.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-400.png,Lung_Opacity/images/Lung_Opacity-400.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-400.png,Lung_Opacity/masks/Lung_Opacity-400.png,True,False,299,299,L,97.27,b6b036192ef599051d1ef0c032581f0c
Lung_Opacity-4000.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4000.png,Lung_Opacity/images/Lung_Opacity-4000.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4000.png,Lung_Opacity/masks/Lung_Opacity-4000.png,True,False,299,299,L,116.35,8941ce63c8cdc5e4eb6dce3e2d191fa9
Lung_Opacity-4001.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4001.png,Lung_Opacity/images/Lung_Opacity-4001.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4001.png,Lung_Opacity/masks/Lung_Opacity-4001.png,True,False,299,299,L,127.83,a7f594895bf37327c2077070d1e5339d
Lung_Opacity-4002.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4002.png,Lung_Opacity/images/Lung_Opacity-4002.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4002.png,Lung_Opacity/masks/Lung_Opacity-4002.png,True,False,299,299,L,144.73,ef45c8c69eeabd8c904f7d0bbb191aec
Lung_Opacity-4003.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4003.png,Lung_Opacity/images/Lung_Opacity-4003.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4003.png,Lung_Opacity/masks/Lung_Opacity-4003.png,True,False,299,299,L,155.54,3454c3391dfc8e96c07caec691d5b76f
Lung_Opacity-4004.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4004.png,Lung_Opacity/images/Lung_Opacity-4004.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4004.png,Lung_Opacity/masks/Lung_Opacity-4004.png,True,False,299,299,L,103.37,0947685f2526a15c49898b3242c5c66c
Lung_Opacity-4005.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4005.png,Lung_Opacity/images/Lung_Opacity-4005.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4005.png,Lung_Opacity/masks/Lung_Opacity-4005.png,True,False,299,299,L,149.53,652c837b848a04ea1fee2e33f1309f3d
Lung_Opacity-4006.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4006.png,Lung_Opacity/images/Lung_Opacity-4006.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4006.png,Lung_Opacity/masks/Lung_Opacity-4006.png,True,False,299,299,L,114.39,2b2c21dea19aef56086226c5efc2538b
Lung_Opacity-4007.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4007.png,Lung_Opacity/images/Lung_Opacity-4007.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4007.png,Lung_Opacity/masks/Lung_Opacity-4007.png,True,False,299,299,L,147.64,553719c124bf2e58ab28512028cc88b5
Lung_Opacity-4008.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4008.png,Lung_Opacity/images/Lung_Opacity-4008.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4008.png,Lung_Opacity/masks/Lung_Opacity-4008.png,True,False,299,299,L,164.93,5a94372b89d6872746344e1978267f58
Lung_Opacity-4009.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4009.png,Lung_Opacity/images/Lung_Opacity-4009.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4009.png,Lung_Opacity/masks/Lung_Opacity-4009.png,True,False,299,299,L,115.09,24253a2765c5673d168d07ab0d902ae9
Lung_Opacity-401.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-401.png,Lung_Opacity/images/Lung_Opacity-401.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-401.png,Lung_Opacity/masks/Lung_Opacity-401.png,True,False,299,299,L,160.41,8457059672a132e7a91707b1dee11bc0
Lung_Opacity-4010.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4010.png,Lung_Opacity/images/Lung_Opacity-4010.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4010.png,Lung_Opacity/masks/Lung_Opacity-4010.png,True,False,299,299,L,130.44,f66bf708c20ca8b8ea46c01fde597884
Lung_Opacity-4011.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4011.png,Lung_Opacity/images/Lung_Opacity-4011.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4011.png,Lung_Opacity/masks/Lung_Opacity-4011.png,True,False,299,299,L,111.7,c41d9d2eec99fdbcf7af3cdfcb859718
Lung_Opacity-4012.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4012.png,Lung_Opacity/images/Lung_Opacity-4012.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4012.png,Lung_Opacity/masks/Lung_Opacity-4012.png,True,False,299,299,L,134.38,04ed1c5be9b59320855cc68432b176a0
Lung_Opacity-4013.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4013.png,Lung_Opacity/images/Lung_Opacity-4013.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4013.png,Lung_Opacity/masks/Lung_Opacity-4013.png,True,False,299,299,L,122.35,3f68713daa963fc21ebfff2c0b0e9239
Lung_Opacity-4014.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4014.png,Lung_Opacity/images/Lung_Opacity-4014.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4014.png,Lung_Opacity/masks/Lung_Opacity-4014.png,True,False,299,299,L,101.3,d75d3ade6f84e6d8b94419a1d629136b
Lung_Opacity-4015.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4015.png,Lung_Opacity/images/Lung_Opacity-4015.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4015.png,Lung_Opacity/masks/Lung_Opacity-4015.png,True,False,299,299,L,95.49,ee5446c6aa07dc3d01c455d4df94d3b0
Lung_Opacity-4016.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4016.png,Lung_Opacity/images/Lung_Opacity-4016.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4016.png,Lung_Opacity/masks/Lung_Opacity-4016.png,True,False,299,299,L,125.84,1f74512d8b168b3cacde8ef703d7c0f5
Lung_Opacity-4017.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4017.png,Lung_Opacity/images/Lung_Opacity-4017.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4017.png,Lung_Opacity/masks/Lung_Opacity-4017.png,True,False,299,299,L,119.12,bd816c7a5d99eaaa0cc3835c59aae338
Lung_Opacity-4018.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4018.png,Lung_Opacity/images/Lung_Opacity-4018.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4018.png,Lung_Opacity/masks/Lung_Opacity-4018.png,True,False,299,299,L,165.45,18b05b4acc4cbd575444fc7e70cbe850
Lung_Opacity-4019.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4019.png,Lung_Opacity/images/Lung_Opacity-4019.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4019.png,Lung_Opacity/masks/Lung_Opacity-4019.png,True,False,299,299,L,100.36,effa41c6fb90d48b9d337d1c5dae1acd
Lung_Opacity-402.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-402.png,Lung_Opacity/images/Lung_Opacity-402.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-402.png,Lung_Opacity/masks/Lung_Opacity-402.png,True,False,299,299,L,90.89,c7173787b8b42ccdda3e8c38001be000
Lung_Opacity-4020.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4020.png,Lung_Opacity/images/Lung_Opacity-4020.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4020.png,Lung_Opacity/masks/Lung_Opacity-4020.png,True,False,299,299,L,125.02,e3c99fac12575df620f893a6ea1c2b9d
Lung_Opacity-4021.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4021.png,Lung_Opacity/images/Lung_Opacity-4021.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4021.png,Lung_Opacity/masks/Lung_Opacity-4021.png,True,False,299,299,L,115.31,9601d78a2648af9e882ea52525c5515f
Lung_Opacity-4022.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4022.png,Lung_Opacity/images/Lung_Opacity-4022.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4022.png,Lung_Opacity/masks/Lung_Opacity-4022.png,True,False,299,299,L,151.39,d106632897156d4598088615f4d70bad
Lung_Opacity-4023.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4023.png,Lung_Opacity/images/Lung_Opacity-4023.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4023.png,Lung_Opacity/masks/Lung_Opacity-4023.png,True,False,299,299,L,118.67,8fbd91bd91e20c045f5c35615349333d
Lung_Opacity-4024.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4024.png,Lung_Opacity/images/Lung_Opacity-4024.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4024.png,Lung_Opacity/masks/Lung_Opacity-4024.png,True,False,299,299,L,141.45,6715ead89c36c4f7cc47c2d5839ab348
Lung_Opacity-4025.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4025.png,Lung_Opacity/images/Lung_Opacity-4025.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4025.png,Lung_Opacity/masks/Lung_Opacity-4025.png,True,False,299,299,L,130.82,15ccc33562cf15e5c42f3e1eaf64ad37
Lung_Opacity-4026.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4026.png,Lung_Opacity/images/Lung_Opacity-4026.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4026.png,Lung_Opacity/masks/Lung_Opacity-4026.png,True,False,299,299,L,146.2,9d2ecf181122bef6aaa9990448566f53
Lung_Opacity-4027.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4027.png,Lung_Opacity/images/Lung_Opacity-4027.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4027.png,Lung_Opacity/masks/Lung_Opacity-4027.png,True,False,299,299,L,162.63,60c1f8fe35e3d12c60b1b920c6c454c8
Lung_Opacity-4028.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4028.png,Lung_Opacity/images/Lung_Opacity-4028.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4028.png,Lung_Opacity/masks/Lung_Opacity-4028.png,True,False,299,299,L,161.07,bf43cbb9b49e1bd6bb96352b63de59e5
Lung_Opacity-4029.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4029.png,Lung_Opacity/images/Lung_Opacity-4029.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4029.png,Lung_Opacity/masks/Lung_Opacity-4029.png,True,False,299,299,L,118.46,e0a1ac0069f9d4bef1d2f5301efe29f8
Lung_Opacity-403.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-403.png,Lung_Opacity/images/Lung_Opacity-403.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-403.png,Lung_Opacity/masks/Lung_Opacity-403.png,True,False,299,299,L,123.42,68f2656767857cb3c274e27a9dc5eadf
Lung_Opacity-4030.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4030.png,Lung_Opacity/images/Lung_Opacity-4030.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4030.png,Lung_Opacity/masks/Lung_Opacity-4030.png,True,False,299,299,L,124.74,2a863d5faae55f9b32e69aa1d699847c
Lung_Opacity-4031.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4031.png,Lung_Opacity/images/Lung_Opacity-4031.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4031.png,Lung_Opacity/masks/Lung_Opacity-4031.png,True,False,299,299,L,118.17,513177c82d3f4da0e4ec6af6c7ba8afd
Lung_Opacity-4032.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4032.png,Lung_Opacity/images/Lung_Opacity-4032.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4032.png,Lung_Opacity/masks/Lung_Opacity-4032.png,True,False,299,299,L,169.71,3aadf532b92c46cad36b1e111c8f7b4e
Lung_Opacity-4033.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4033.png,Lung_Opacity/images/Lung_Opacity-4033.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4033.png,Lung_Opacity/masks/Lung_Opacity-4033.png,True,False,299,299,L,111.35,cb719a1d07a3fa720b76d15c726496c1
Lung_Opacity-4034.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4034.png,Lung_Opacity/images/Lung_Opacity-4034.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4034.png,Lung_Opacity/masks/Lung_Opacity-4034.png,True,False,299,299,L,153.51,e9d085e38b0a67ac86dfeeb4640472a9
Lung_Opacity-4035.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4035.png,Lung_Opacity/images/Lung_Opacity-4035.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4035.png,Lung_Opacity/masks/Lung_Opacity-4035.png,True,False,299,299,L,113.71,22a14fd3106f3fb16a3e428e552cd633
Lung_Opacity-4036.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4036.png,Lung_Opacity/images/Lung_Opacity-4036.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4036.png,Lung_Opacity/masks/Lung_Opacity-4036.png,True,False,299,299,L,170.2,4df7ddbca3e223af2f154e076efbaa30
Lung_Opacity-4037.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4037.png,Lung_Opacity/images/Lung_Opacity-4037.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4037.png,Lung_Opacity/masks/Lung_Opacity-4037.png,True,False,299,299,L,94.66,9df58541ae3fb2c917daae378fb2bf1b
Lung_Opacity-4038.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4038.png,Lung_Opacity/images/Lung_Opacity-4038.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4038.png,Lung_Opacity/masks/Lung_Opacity-4038.png,True,False,299,299,L,98.17,9473713872a7cc0f173857336d9c6b9f
Lung_Opacity-4039.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4039.png,Lung_Opacity/images/Lung_Opacity-4039.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4039.png,Lung_Opacity/masks/Lung_Opacity-4039.png,True,False,299,299,L,102.99,c5a893f387dae2015fee6cff371e018b
Lung_Opacity-404.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-404.png,Lung_Opacity/images/Lung_Opacity-404.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-404.png,Lung_Opacity/masks/Lung_Opacity-404.png,True,False,299,299,L,166.94,174635ff2619f3780a08bf41b430fea9
Lung_Opacity-4040.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4040.png,Lung_Opacity/images/Lung_Opacity-4040.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4040.png,Lung_Opacity/masks/Lung_Opacity-4040.png,True,False,299,299,L,121.75,8e971af13afd005c6d86467794956589
Lung_Opacity-4041.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4041.png,Lung_Opacity/images/Lung_Opacity-4041.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4041.png,Lung_Opacity/masks/Lung_Opacity-4041.png,True,False,299,299,L,120.53,6eff9fcb2dac780c60240c95c274a198
Lung_Opacity-4042.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4042.png,Lung_Opacity/images/Lung_Opacity-4042.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4042.png,Lung_Opacity/masks/Lung_Opacity-4042.png,True,False,299,299,L,174.12,0b4f6bc5dfd3d87d95d84931af306576
Lung_Opacity-4043.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4043.png,Lung_Opacity/images/Lung_Opacity-4043.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4043.png,Lung_Opacity/masks/Lung_Opacity-4043.png,True,False,299,299,L,132.71,6aba1d0ea3a4a22c3e66f37a50ec87a4
Lung_Opacity-4044.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4044.png,Lung_Opacity/images/Lung_Opacity-4044.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4044.png,Lung_Opacity/masks/Lung_Opacity-4044.png,True,False,299,299,L,118.13,c905bb0faf09cd250587285bfed31d85
Lung_Opacity-4045.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4045.png,Lung_Opacity/images/Lung_Opacity-4045.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4045.png,Lung_Opacity/masks/Lung_Opacity-4045.png,True,False,299,299,L,112.12,7178b8eea8e5fe0d23b57899305a90db
Lung_Opacity-4046.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4046.png,Lung_Opacity/images/Lung_Opacity-4046.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4046.png,Lung_Opacity/masks/Lung_Opacity-4046.png,True,False,299,299,L,128.05,f33b2d95a45fdf5cd88b24f7720a1b82
Lung_Opacity-4047.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4047.png,Lung_Opacity/images/Lung_Opacity-4047.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4047.png,Lung_Opacity/masks/Lung_Opacity-4047.png,True,False,299,299,L,107.05,7851d6cbfe3e54930616c905edeafc58
Lung_Opacity-4048.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4048.png,Lung_Opacity/images/Lung_Opacity-4048.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4048.png,Lung_Opacity/masks/Lung_Opacity-4048.png,True,False,299,299,L,91.52,5e461a5b7141a437c0fe5855e8cbd78f
Lung_Opacity-4049.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4049.png,Lung_Opacity/images/Lung_Opacity-4049.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4049.png,Lung_Opacity/masks/Lung_Opacity-4049.png,True,False,299,299,L,156.82,90da3a7ee506acdd5c510d78cd6d8126
Lung_Opacity-405.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-405.png,Lung_Opacity/images/Lung_Opacity-405.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-405.png,Lung_Opacity/masks/Lung_Opacity-405.png,True,False,299,299,L,121.04,9ec731e666efadbd162911699de6d865
Lung_Opacity-4050.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4050.png,Lung_Opacity/images/Lung_Opacity-4050.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4050.png,Lung_Opacity/masks/Lung_Opacity-4050.png,True,False,299,299,L,146.85,df2ea5962c60d4408099c70c126140d5
Lung_Opacity-4051.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4051.png,Lung_Opacity/images/Lung_Opacity-4051.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4051.png,Lung_Opacity/masks/Lung_Opacity-4051.png,True,False,299,299,L,107.58,e54350d60b91d529236841e704823e40
Lung_Opacity-4052.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4052.png,Lung_Opacity/images/Lung_Opacity-4052.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4052.png,Lung_Opacity/masks/Lung_Opacity-4052.png,True,False,299,299,L,121.47,a0386114fbba985f9cfb783b5f099585
Lung_Opacity-4053.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4053.png,Lung_Opacity/images/Lung_Opacity-4053.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4053.png,Lung_Opacity/masks/Lung_Opacity-4053.png,True,False,299,299,L,93.87,70d420b8cfe5ee4a616c39f796d4f8b9
Lung_Opacity-4054.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4054.png,Lung_Opacity/images/Lung_Opacity-4054.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4054.png,Lung_Opacity/masks/Lung_Opacity-4054.png,True,False,299,299,L,159.03,53f0aa4ce42254fd9735cd53107af1e7
Lung_Opacity-4055.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4055.png,Lung_Opacity/images/Lung_Opacity-4055.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4055.png,Lung_Opacity/masks/Lung_Opacity-4055.png,True,False,299,299,L,111.92,deaaa42b22ca478ca08d23f63e0cfee5
Lung_Opacity-4056.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4056.png,Lung_Opacity/images/Lung_Opacity-4056.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4056.png,Lung_Opacity/masks/Lung_Opacity-4056.png,True,False,299,299,L,108.92,28451667505f495cb7d801a1c093e222
Lung_Opacity-4057.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4057.png,Lung_Opacity/images/Lung_Opacity-4057.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4057.png,Lung_Opacity/masks/Lung_Opacity-4057.png,True,False,299,299,L,146.77,f627dd1c32e9a01e48895290c6116b66
Lung_Opacity-4058.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4058.png,Lung_Opacity/images/Lung_Opacity-4058.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4058.png,Lung_Opacity/masks/Lung_Opacity-4058.png,True,False,299,299,L,126.6,b843ab1ddc8348684464156fbaa3eeb5
Lung_Opacity-4059.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4059.png,Lung_Opacity/images/Lung_Opacity-4059.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4059.png,Lung_Opacity/masks/Lung_Opacity-4059.png,True,False,299,299,L,132.38,7b1d381631ec6852842b8e0e9697abd6
Lung_Opacity-406.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-406.png,Lung_Opacity/images/Lung_Opacity-406.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-406.png,Lung_Opacity/masks/Lung_Opacity-406.png,True,False,299,299,L,118.1,b1095014ace30f66f9c40f4f8e6e54b5
Lung_Opacity-4060.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4060.png,Lung_Opacity/images/Lung_Opacity-4060.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4060.png,Lung_Opacity/masks/Lung_Opacity-4060.png,True,False,299,299,L,161.56,e7db813fbee1c598a459b74d4e5f6e94
Lung_Opacity-4061.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4061.png,Lung_Opacity/images/Lung_Opacity-4061.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4061.png,Lung_Opacity/masks/Lung_Opacity-4061.png,True,False,299,299,L,169.78,f910b570dfe228f93f6ace00448794d6
Lung_Opacity-4062.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4062.png,Lung_Opacity/images/Lung_Opacity-4062.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4062.png,Lung_Opacity/masks/Lung_Opacity-4062.png,True,False,299,299,L,154.25,adde31e54624fc909c578a40f63943a3
Lung_Opacity-4063.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4063.png,Lung_Opacity/images/Lung_Opacity-4063.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4063.png,Lung_Opacity/masks/Lung_Opacity-4063.png,True,False,299,299,L,107.31,8fe287c5025dee6f6cdf3e19847dc1c4
Lung_Opacity-4064.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4064.png,Lung_Opacity/images/Lung_Opacity-4064.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4064.png,Lung_Opacity/masks/Lung_Opacity-4064.png,True,False,299,299,L,166.53,66af0fdcf3b201422d2f40e6aca1c665
Lung_Opacity-4065.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4065.png,Lung_Opacity/images/Lung_Opacity-4065.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4065.png,Lung_Opacity/masks/Lung_Opacity-4065.png,True,False,299,299,L,177.41,cbc769c9456cdeb8652482e159b01d88
Lung_Opacity-4066.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4066.png,Lung_Opacity/images/Lung_Opacity-4066.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4066.png,Lung_Opacity/masks/Lung_Opacity-4066.png,True,False,299,299,L,132.7,8a933155ade7decf8bd7407fd9441457
Lung_Opacity-4067.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4067.png,Lung_Opacity/images/Lung_Opacity-4067.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4067.png,Lung_Opacity/masks/Lung_Opacity-4067.png,True,False,299,299,L,91.28,5bd4f5dc70ba23aa6668ee2f464f9fee
Lung_Opacity-4068.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4068.png,Lung_Opacity/images/Lung_Opacity-4068.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4068.png,Lung_Opacity/masks/Lung_Opacity-4068.png,True,False,299,299,L,105.79,f71c52292ea56687f9477b01a766dea1
Lung_Opacity-4069.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4069.png,Lung_Opacity/images/Lung_Opacity-4069.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4069.png,Lung_Opacity/masks/Lung_Opacity-4069.png,True,False,299,299,L,114.92,a201374543ff8d3e6890cb187d30816f
Lung_Opacity-407.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-407.png,Lung_Opacity/images/Lung_Opacity-407.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-407.png,Lung_Opacity/masks/Lung_Opacity-407.png,True,False,299,299,L,89.55,40e74fe7ac011fe38fdcfe6803c15b5b
Lung_Opacity-4070.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4070.png,Lung_Opacity/images/Lung_Opacity-4070.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4070.png,Lung_Opacity/masks/Lung_Opacity-4070.png,True,False,299,299,L,97.65,bbb33d5af4e95fcbff4c9876dbe370e8
Lung_Opacity-4071.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4071.png,Lung_Opacity/images/Lung_Opacity-4071.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4071.png,Lung_Opacity/masks/Lung_Opacity-4071.png,True,False,299,299,L,98.87,b8321bad8e66abf3dc83a62d8ca7a1a6
Lung_Opacity-4072.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4072.png,Lung_Opacity/images/Lung_Opacity-4072.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4072.png,Lung_Opacity/masks/Lung_Opacity-4072.png,True,False,299,299,L,122.72,74a7702d9b94c8c3664a86161b11e8ac
Lung_Opacity-4073.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4073.png,Lung_Opacity/images/Lung_Opacity-4073.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4073.png,Lung_Opacity/masks/Lung_Opacity-4073.png,True,False,299,299,L,152.39,530c1005b72e552eebbcc7c5bcc1f30b
Lung_Opacity-4074.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4074.png,Lung_Opacity/images/Lung_Opacity-4074.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4074.png,Lung_Opacity/masks/Lung_Opacity-4074.png,True,False,299,299,L,111.85,03ccd7ff4bc73ef4376f5a1a1f7f89a8
Lung_Opacity-4075.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4075.png,Lung_Opacity/images/Lung_Opacity-4075.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4075.png,Lung_Opacity/masks/Lung_Opacity-4075.png,True,False,299,299,L,120.67,150296b9c44de9604326417cbe784032
Lung_Opacity-4076.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4076.png,Lung_Opacity/images/Lung_Opacity-4076.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4076.png,Lung_Opacity/masks/Lung_Opacity-4076.png,True,False,299,299,L,121.52,ecb5288b43aa183fb33c549065e1e00f
Lung_Opacity-4077.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4077.png,Lung_Opacity/images/Lung_Opacity-4077.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4077.png,Lung_Opacity/masks/Lung_Opacity-4077.png,True,False,299,299,L,129.8,0360b4f2b623a12f82ffe779203a2952
Lung_Opacity-4078.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4078.png,Lung_Opacity/images/Lung_Opacity-4078.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4078.png,Lung_Opacity/masks/Lung_Opacity-4078.png,True,False,299,299,L,133.15,b7e2f7a37f3ab75a8cc2d1edda062c9a
Lung_Opacity-4079.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4079.png,Lung_Opacity/images/Lung_Opacity-4079.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4079.png,Lung_Opacity/masks/Lung_Opacity-4079.png,True,False,299,299,L,159.7,1e4e7643d766a8cbd2d7ca2ed580aac8
Lung_Opacity-408.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-408.png,Lung_Opacity/images/Lung_Opacity-408.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-408.png,Lung_Opacity/masks/Lung_Opacity-408.png,True,False,299,299,L,135.13,293b1bee0469538c3eba39541b7fc4a9
Lung_Opacity-4080.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4080.png,Lung_Opacity/images/Lung_Opacity-4080.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4080.png,Lung_Opacity/masks/Lung_Opacity-4080.png,True,False,299,299,L,162.07,d4ae91084df1ddeddf2d65b965e65b20
Lung_Opacity-4081.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4081.png,Lung_Opacity/images/Lung_Opacity-4081.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4081.png,Lung_Opacity/masks/Lung_Opacity-4081.png,True,False,299,299,L,139.4,e344cf1b4103c8edec7d7b015c80127d
Lung_Opacity-4082.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4082.png,Lung_Opacity/images/Lung_Opacity-4082.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4082.png,Lung_Opacity/masks/Lung_Opacity-4082.png,True,False,299,299,L,106.8,c628c5d38028c9e02144d81b2097e7e2
Lung_Opacity-4083.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4083.png,Lung_Opacity/images/Lung_Opacity-4083.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4083.png,Lung_Opacity/masks/Lung_Opacity-4083.png,True,False,299,299,L,154.57,52886a08348a9adf8c37805cfccbd566
Lung_Opacity-4084.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4084.png,Lung_Opacity/images/Lung_Opacity-4084.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4084.png,Lung_Opacity/masks/Lung_Opacity-4084.png,True,False,299,299,L,111.9,7a8f5f6d7fdb5c907e4ba66615525e85
Lung_Opacity-4085.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4085.png,Lung_Opacity/images/Lung_Opacity-4085.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4085.png,Lung_Opacity/masks/Lung_Opacity-4085.png,True,False,299,299,L,111.75,04af5b4a1162cba645b3033be54a22e1
Lung_Opacity-4086.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4086.png,Lung_Opacity/images/Lung_Opacity-4086.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4086.png,Lung_Opacity/masks/Lung_Opacity-4086.png,True,False,299,299,L,116.29,4cf663d316583a4e1d4a440e5f5f216f
Lung_Opacity-4087.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4087.png,Lung_Opacity/images/Lung_Opacity-4087.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4087.png,Lung_Opacity/masks/Lung_Opacity-4087.png,True,False,299,299,L,106.33,505314ccadcb796bde322b0c862792a2
Lung_Opacity-4088.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4088.png,Lung_Opacity/images/Lung_Opacity-4088.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4088.png,Lung_Opacity/masks/Lung_Opacity-4088.png,True,False,299,299,L,118.86,2d1c55c6b48401a1d4103b8242651956
Lung_Opacity-4089.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4089.png,Lung_Opacity/images/Lung_Opacity-4089.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4089.png,Lung_Opacity/masks/Lung_Opacity-4089.png,True,False,299,299,L,127.89,6b7720a93d0de0b837e2a37b29a9f20e
Lung_Opacity-409.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-409.png,Lung_Opacity/images/Lung_Opacity-409.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-409.png,Lung_Opacity/masks/Lung_Opacity-409.png,True,False,299,299,L,105.71,f02769ffed466c5bf2a1e41682c0a490
Lung_Opacity-4090.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4090.png,Lung_Opacity/images/Lung_Opacity-4090.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4090.png,Lung_Opacity/masks/Lung_Opacity-4090.png,True,False,299,299,L,95.81,addae28c8cae0ebce319f6e1db414029
Lung_Opacity-4091.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4091.png,Lung_Opacity/images/Lung_Opacity-4091.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4091.png,Lung_Opacity/masks/Lung_Opacity-4091.png,True,False,299,299,L,153.14,c55468d9c0f5879678e9e038972d3cf6
Lung_Opacity-4092.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4092.png,Lung_Opacity/images/Lung_Opacity-4092.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4092.png,Lung_Opacity/masks/Lung_Opacity-4092.png,True,False,299,299,L,96.17,0efe0ff24f781e46665085b79ecf1808
Lung_Opacity-4093.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4093.png,Lung_Opacity/images/Lung_Opacity-4093.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4093.png,Lung_Opacity/masks/Lung_Opacity-4093.png,True,False,299,299,L,138.79,e57e3718a4d9f67d4505a02963286df7
Lung_Opacity-4094.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4094.png,Lung_Opacity/images/Lung_Opacity-4094.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4094.png,Lung_Opacity/masks/Lung_Opacity-4094.png,True,False,299,299,L,176.97,baa9dcb210b22eb60dbbc0f4430955c4
Lung_Opacity-4095.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4095.png,Lung_Opacity/images/Lung_Opacity-4095.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4095.png,Lung_Opacity/masks/Lung_Opacity-4095.png,True,False,299,299,L,131.94,3ba6356bd651c9afa919dc4c95e70663
Lung_Opacity-4096.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4096.png,Lung_Opacity/images/Lung_Opacity-4096.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4096.png,Lung_Opacity/masks/Lung_Opacity-4096.png,True,False,299,299,L,146.29,d8198dd80a039f58bfba91a32d85975c
Lung_Opacity-4097.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4097.png,Lung_Opacity/images/Lung_Opacity-4097.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4097.png,Lung_Opacity/masks/Lung_Opacity-4097.png,True,False,299,299,L,115.7,2d8bc2cf3368992e67fe155ac6a8c882
Lung_Opacity-4098.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4098.png,Lung_Opacity/images/Lung_Opacity-4098.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4098.png,Lung_Opacity/masks/Lung_Opacity-4098.png,True,False,299,299,L,139.77,62a731cad1df15f07c37f0fc9b01dc25
Lung_Opacity-4099.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4099.png,Lung_Opacity/images/Lung_Opacity-4099.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4099.png,Lung_Opacity/masks/Lung_Opacity-4099.png,True,False,299,299,L,107.2,f55cb3d04e4c7ec088fb83fdcb310084
Lung_Opacity-41.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-41.png,Lung_Opacity/images/Lung_Opacity-41.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-41.png,Lung_Opacity/masks/Lung_Opacity-41.png,True,False,299,299,L,121.17,9d7b047183dd98ae763feadd24389c7b
Lung_Opacity-410.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-410.png,Lung_Opacity/images/Lung_Opacity-410.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-410.png,Lung_Opacity/masks/Lung_Opacity-410.png,True,False,299,299,L,177.52,37f3bf268c72ba9de8755ea6127fedde
Lung_Opacity-4100.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4100.png,Lung_Opacity/images/Lung_Opacity-4100.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4100.png,Lung_Opacity/masks/Lung_Opacity-4100.png,True,False,299,299,L,110.55,cb110e4f20de9d2fa39c841cab1d56b9
Lung_Opacity-4101.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4101.png,Lung_Opacity/images/Lung_Opacity-4101.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4101.png,Lung_Opacity/masks/Lung_Opacity-4101.png,True,False,299,299,L,143.61,d65bcab1dcdb13ddb20cf7370a19a95b
Lung_Opacity-4102.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4102.png,Lung_Opacity/images/Lung_Opacity-4102.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4102.png,Lung_Opacity/masks/Lung_Opacity-4102.png,True,False,299,299,L,128.1,fd40eb8cb87345f5e628d2b8cf823347
Lung_Opacity-4103.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4103.png,Lung_Opacity/images/Lung_Opacity-4103.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4103.png,Lung_Opacity/masks/Lung_Opacity-4103.png,True,False,299,299,L,45.25,b52f824d4c919b6a4080c802e14b7f4b
Lung_Opacity-4104.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4104.png,Lung_Opacity/images/Lung_Opacity-4104.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4104.png,Lung_Opacity/masks/Lung_Opacity-4104.png,True,False,299,299,L,115.38,f232a08fe2146589c72ac969a154b9d9
Lung_Opacity-4105.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4105.png,Lung_Opacity/images/Lung_Opacity-4105.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4105.png,Lung_Opacity/masks/Lung_Opacity-4105.png,True,False,299,299,L,124.29,ebe8cbaf1dfe2c085e4500f4a3ea4f3b
Lung_Opacity-4106.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4106.png,Lung_Opacity/images/Lung_Opacity-4106.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4106.png,Lung_Opacity/masks/Lung_Opacity-4106.png,True,False,299,299,L,157.65,7d48ca793478902a3fcc12789df169d9
Lung_Opacity-4107.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4107.png,Lung_Opacity/images/Lung_Opacity-4107.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4107.png,Lung_Opacity/masks/Lung_Opacity-4107.png,True,False,299,299,L,106.29,98eb613c8f10df1b9a8acb6274257418
Lung_Opacity-4108.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4108.png,Lung_Opacity/images/Lung_Opacity-4108.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4108.png,Lung_Opacity/masks/Lung_Opacity-4108.png,True,False,299,299,L,117.58,c92ad7e4feb84a4e021223d1d112eea6
Lung_Opacity-4109.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4109.png,Lung_Opacity/images/Lung_Opacity-4109.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4109.png,Lung_Opacity/masks/Lung_Opacity-4109.png,True,False,299,299,L,106.28,83c479cbe45eddb1b80333dd07897c67
Lung_Opacity-411.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-411.png,Lung_Opacity/images/Lung_Opacity-411.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-411.png,Lung_Opacity/masks/Lung_Opacity-411.png,True,False,299,299,L,94.49,0b9d2431320b6380c786113e9d3adbd1
Lung_Opacity-4110.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4110.png,Lung_Opacity/images/Lung_Opacity-4110.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4110.png,Lung_Opacity/masks/Lung_Opacity-4110.png,True,False,299,299,L,85.99,3c54e51d43dc8d57cc47dfa7441fdb10
Lung_Opacity-4111.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4111.png,Lung_Opacity/images/Lung_Opacity-4111.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4111.png,Lung_Opacity/masks/Lung_Opacity-4111.png,True,False,299,299,L,155.77,b3433d16685e1b7dfefea43a95160d51
Lung_Opacity-4112.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4112.png,Lung_Opacity/images/Lung_Opacity-4112.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4112.png,Lung_Opacity/masks/Lung_Opacity-4112.png,True,False,299,299,L,72.23,0b21850ef65882a62d694b589d3617ba
Lung_Opacity-4113.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4113.png,Lung_Opacity/images/Lung_Opacity-4113.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4113.png,Lung_Opacity/masks/Lung_Opacity-4113.png,True,False,299,299,L,135.23,df89ba726f19ca750ee3bdab8a13a48d
Lung_Opacity-4114.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4114.png,Lung_Opacity/images/Lung_Opacity-4114.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4114.png,Lung_Opacity/masks/Lung_Opacity-4114.png,True,False,299,299,L,105.49,14f8338df516bdd67878d71442b87670
Lung_Opacity-4115.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4115.png,Lung_Opacity/images/Lung_Opacity-4115.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4115.png,Lung_Opacity/masks/Lung_Opacity-4115.png,True,False,299,299,L,109.96,70e267df4b7d68acb171cbcd07ed36dd
Lung_Opacity-4116.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4116.png,Lung_Opacity/images/Lung_Opacity-4116.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4116.png,Lung_Opacity/masks/Lung_Opacity-4116.png,True,False,299,299,L,164.05,976791bade2c1fb42933643aaf7e8535
Lung_Opacity-4117.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4117.png,Lung_Opacity/images/Lung_Opacity-4117.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4117.png,Lung_Opacity/masks/Lung_Opacity-4117.png,True,False,299,299,L,118.46,e83dfd21b78d924e4d16f0c9f3eb6b2d
Lung_Opacity-4118.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4118.png,Lung_Opacity/images/Lung_Opacity-4118.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4118.png,Lung_Opacity/masks/Lung_Opacity-4118.png,True,False,299,299,L,118.02,df62dfebebe666e0fdf43ce64ed402c0
Lung_Opacity-4119.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4119.png,Lung_Opacity/images/Lung_Opacity-4119.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4119.png,Lung_Opacity/masks/Lung_Opacity-4119.png,True,False,299,299,L,99.28,07d976f838ac4184bc8372db29f0d362
Lung_Opacity-412.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-412.png,Lung_Opacity/images/Lung_Opacity-412.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-412.png,Lung_Opacity/masks/Lung_Opacity-412.png,True,False,299,299,L,146.79,ae4707f54b7247291820593046b84910
Lung_Opacity-4120.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4120.png,Lung_Opacity/images/Lung_Opacity-4120.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4120.png,Lung_Opacity/masks/Lung_Opacity-4120.png,True,False,299,299,L,173.5,ca22605161f32ecbc67698726e450691
Lung_Opacity-4121.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4121.png,Lung_Opacity/images/Lung_Opacity-4121.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4121.png,Lung_Opacity/masks/Lung_Opacity-4121.png,True,False,299,299,L,130.66,7ee94120fd781e01f12c5193dc23b1f9
Lung_Opacity-4122.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4122.png,Lung_Opacity/images/Lung_Opacity-4122.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4122.png,Lung_Opacity/masks/Lung_Opacity-4122.png,True,False,299,299,L,175.26,2a0ffb88fa7e1879f03fa537947f51c8
Lung_Opacity-4123.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4123.png,Lung_Opacity/images/Lung_Opacity-4123.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4123.png,Lung_Opacity/masks/Lung_Opacity-4123.png,True,False,299,299,L,170.38,aadcec5bc19b5f98e033444d2a379da0
Lung_Opacity-4124.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4124.png,Lung_Opacity/images/Lung_Opacity-4124.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4124.png,Lung_Opacity/masks/Lung_Opacity-4124.png,True,False,299,299,L,170.61,18f72b4685b940068fc498cb9229f840
Lung_Opacity-4125.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4125.png,Lung_Opacity/images/Lung_Opacity-4125.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4125.png,Lung_Opacity/masks/Lung_Opacity-4125.png,True,False,299,299,L,117.93,7d2a08ce6bf0f878d8f46ab54586bcd5
Lung_Opacity-4126.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4126.png,Lung_Opacity/images/Lung_Opacity-4126.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4126.png,Lung_Opacity/masks/Lung_Opacity-4126.png,True,False,299,299,L,119.51,feece2d2ade919251cc0ebe91d87cc82
Lung_Opacity-4127.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4127.png,Lung_Opacity/images/Lung_Opacity-4127.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4127.png,Lung_Opacity/masks/Lung_Opacity-4127.png,True,False,299,299,L,89.78,f4a746967ecb92e1e863792e8b8bcea3
Lung_Opacity-4128.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4128.png,Lung_Opacity/images/Lung_Opacity-4128.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4128.png,Lung_Opacity/masks/Lung_Opacity-4128.png,True,False,299,299,L,110.35,3b7dd66439c8372a3595b809a1e18ae2
Lung_Opacity-4129.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4129.png,Lung_Opacity/images/Lung_Opacity-4129.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4129.png,Lung_Opacity/masks/Lung_Opacity-4129.png,True,False,299,299,L,129.22,16ef33aaae06752df5ffac8555cb9224
Lung_Opacity-413.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-413.png,Lung_Opacity/images/Lung_Opacity-413.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-413.png,Lung_Opacity/masks/Lung_Opacity-413.png,True,False,299,299,L,101.22,6ae99c1197ae897a8b2bd2a960a6443a
Lung_Opacity-4130.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4130.png,Lung_Opacity/images/Lung_Opacity-4130.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4130.png,Lung_Opacity/masks/Lung_Opacity-4130.png,True,False,299,299,L,167.02,3af165fa61410483a0a4e9596635186e
Lung_Opacity-4131.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4131.png,Lung_Opacity/images/Lung_Opacity-4131.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4131.png,Lung_Opacity/masks/Lung_Opacity-4131.png,True,False,299,299,L,128.45,ac6151342f3a1a4459972dd9d2eb98cf
Lung_Opacity-4132.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4132.png,Lung_Opacity/images/Lung_Opacity-4132.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4132.png,Lung_Opacity/masks/Lung_Opacity-4132.png,True,False,299,299,L,115.97,8979c0088f124554758357340b289d69
Lung_Opacity-4133.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4133.png,Lung_Opacity/images/Lung_Opacity-4133.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4133.png,Lung_Opacity/masks/Lung_Opacity-4133.png,True,False,299,299,L,142.87,b8f318fb4de1f1f88d87c3c36f77e98d
Lung_Opacity-4134.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4134.png,Lung_Opacity/images/Lung_Opacity-4134.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4134.png,Lung_Opacity/masks/Lung_Opacity-4134.png,True,False,299,299,L,118.25,92110a0fa4971d9f69be124c4c723386
Lung_Opacity-4135.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4135.png,Lung_Opacity/images/Lung_Opacity-4135.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4135.png,Lung_Opacity/masks/Lung_Opacity-4135.png,True,False,299,299,L,125.27,2c0f0342a0e0bd6da466d3a2808d0906
Lung_Opacity-4136.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4136.png,Lung_Opacity/images/Lung_Opacity-4136.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4136.png,Lung_Opacity/masks/Lung_Opacity-4136.png,True,False,299,299,L,124.66,2dc3f8db8bb889e531589cb76a6b126c
Lung_Opacity-4137.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4137.png,Lung_Opacity/images/Lung_Opacity-4137.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4137.png,Lung_Opacity/masks/Lung_Opacity-4137.png,True,False,299,299,L,100.57,9f5361c227c2288a5e3ace3e422e9108
Lung_Opacity-4138.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4138.png,Lung_Opacity/images/Lung_Opacity-4138.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4138.png,Lung_Opacity/masks/Lung_Opacity-4138.png,True,False,299,299,L,109.29,6baf0b19c18d837f15ea6a116426c78d
Lung_Opacity-4139.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4139.png,Lung_Opacity/images/Lung_Opacity-4139.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4139.png,Lung_Opacity/masks/Lung_Opacity-4139.png,True,False,299,299,L,97.14,62fe4b5e36e8c0e6c7529f8ab585deb4
Lung_Opacity-414.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-414.png,Lung_Opacity/images/Lung_Opacity-414.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-414.png,Lung_Opacity/masks/Lung_Opacity-414.png,True,False,299,299,L,117.88,a2ca9cea9db6348cd68e30b09bdae0dc
Lung_Opacity-4140.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4140.png,Lung_Opacity/images/Lung_Opacity-4140.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4140.png,Lung_Opacity/masks/Lung_Opacity-4140.png,True,False,299,299,L,99.93,078d1b4c8682fdbd2cc392dd8230952d
Lung_Opacity-4141.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4141.png,Lung_Opacity/images/Lung_Opacity-4141.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4141.png,Lung_Opacity/masks/Lung_Opacity-4141.png,True,False,299,299,L,165.6,0ddc7861ce99652734d6293dd2f6a169
Lung_Opacity-4142.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4142.png,Lung_Opacity/images/Lung_Opacity-4142.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4142.png,Lung_Opacity/masks/Lung_Opacity-4142.png,True,False,299,299,L,155.67,d07c50f09fe5fab7bbd0a61a9ec82e04
Lung_Opacity-4143.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4143.png,Lung_Opacity/images/Lung_Opacity-4143.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4143.png,Lung_Opacity/masks/Lung_Opacity-4143.png,True,False,299,299,L,110.64,a1ee5ed6deb29f36a2cbb9cfb343dadf
Lung_Opacity-4144.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4144.png,Lung_Opacity/images/Lung_Opacity-4144.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4144.png,Lung_Opacity/masks/Lung_Opacity-4144.png,True,False,299,299,L,110.87,911ead63a75684be87f3aea8169992df
Lung_Opacity-4145.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4145.png,Lung_Opacity/images/Lung_Opacity-4145.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4145.png,Lung_Opacity/masks/Lung_Opacity-4145.png,True,False,299,299,L,127.54,f5916d81fa854c1b5e631b3773ccf2c4
Lung_Opacity-4146.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4146.png,Lung_Opacity/images/Lung_Opacity-4146.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4146.png,Lung_Opacity/masks/Lung_Opacity-4146.png,True,False,299,299,L,109.27,7173c15bd225f20bca8f29104fc95b71
Lung_Opacity-4147.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4147.png,Lung_Opacity/images/Lung_Opacity-4147.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4147.png,Lung_Opacity/masks/Lung_Opacity-4147.png,True,False,299,299,L,169.2,b2281c1d68fd7f0dcc4f02bb792a749e
Lung_Opacity-4148.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4148.png,Lung_Opacity/images/Lung_Opacity-4148.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4148.png,Lung_Opacity/masks/Lung_Opacity-4148.png,True,False,299,299,L,136.22,62ea166f68af9dc482affb9c892080df
Lung_Opacity-4149.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4149.png,Lung_Opacity/images/Lung_Opacity-4149.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4149.png,Lung_Opacity/masks/Lung_Opacity-4149.png,True,False,299,299,L,172.39,8380026f61e97a12fd641afea85f5b2f
Lung_Opacity-415.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-415.png,Lung_Opacity/images/Lung_Opacity-415.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-415.png,Lung_Opacity/masks/Lung_Opacity-415.png,True,False,299,299,L,123.7,46e3199e877d27cc2eef0399e54bb042
Lung_Opacity-4150.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4150.png,Lung_Opacity/images/Lung_Opacity-4150.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4150.png,Lung_Opacity/masks/Lung_Opacity-4150.png,True,False,299,299,L,111.77,4b77e754d2a59d0067a2088fdf8b74f0
Lung_Opacity-4151.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4151.png,Lung_Opacity/images/Lung_Opacity-4151.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4151.png,Lung_Opacity/masks/Lung_Opacity-4151.png,True,False,299,299,L,125.32,b77355012860093505336c92dfccbd8b
Lung_Opacity-4152.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4152.png,Lung_Opacity/images/Lung_Opacity-4152.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4152.png,Lung_Opacity/masks/Lung_Opacity-4152.png,True,False,299,299,L,161.5,56cc88f7ba9f4705d0a260fe060127a2
Lung_Opacity-4153.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4153.png,Lung_Opacity/images/Lung_Opacity-4153.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4153.png,Lung_Opacity/masks/Lung_Opacity-4153.png,True,False,299,299,L,130.82,91317438a04c5b6156d6cedf3c6d003b
Lung_Opacity-4154.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4154.png,Lung_Opacity/images/Lung_Opacity-4154.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4154.png,Lung_Opacity/masks/Lung_Opacity-4154.png,True,False,299,299,L,124.81,b32c169611e8a80abfe28f5023509ac5
Lung_Opacity-4155.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4155.png,Lung_Opacity/images/Lung_Opacity-4155.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4155.png,Lung_Opacity/masks/Lung_Opacity-4155.png,True,False,299,299,L,154.14,32bac00a40f6519e93ef2bca61e861bf
Lung_Opacity-4156.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4156.png,Lung_Opacity/images/Lung_Opacity-4156.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4156.png,Lung_Opacity/masks/Lung_Opacity-4156.png,True,False,299,299,L,111.41,85191027c67c1252e654f80621ff2412
Lung_Opacity-4157.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4157.png,Lung_Opacity/images/Lung_Opacity-4157.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4157.png,Lung_Opacity/masks/Lung_Opacity-4157.png,True,False,299,299,L,111.21,8031c09df1c3bade48675bb78a5628e4
Lung_Opacity-4158.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4158.png,Lung_Opacity/images/Lung_Opacity-4158.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4158.png,Lung_Opacity/masks/Lung_Opacity-4158.png,True,False,299,299,L,104.56,a7f68ce12cf1e97df03b26e5e93c7b74
Lung_Opacity-4159.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4159.png,Lung_Opacity/images/Lung_Opacity-4159.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4159.png,Lung_Opacity/masks/Lung_Opacity-4159.png,True,False,299,299,L,173.87,b084cb2cd8796dbc667d903e4385909a
Lung_Opacity-416.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-416.png,Lung_Opacity/images/Lung_Opacity-416.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-416.png,Lung_Opacity/masks/Lung_Opacity-416.png,True,False,299,299,L,144.43,591b60d6d41dcf9a293d79614d9713ac
Lung_Opacity-4160.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4160.png,Lung_Opacity/images/Lung_Opacity-4160.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4160.png,Lung_Opacity/masks/Lung_Opacity-4160.png,True,False,299,299,L,165.93,4f761a7f89499e68e9326fb870bf0be1
Lung_Opacity-4161.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4161.png,Lung_Opacity/images/Lung_Opacity-4161.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4161.png,Lung_Opacity/masks/Lung_Opacity-4161.png,True,False,299,299,L,98.66,4681d1a2fc91788a002eec0ef8a9d131
Lung_Opacity-4162.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4162.png,Lung_Opacity/images/Lung_Opacity-4162.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4162.png,Lung_Opacity/masks/Lung_Opacity-4162.png,True,False,299,299,L,141.2,a30aa85927b83afb8d0c1aa91e6dea13
Lung_Opacity-4163.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4163.png,Lung_Opacity/images/Lung_Opacity-4163.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4163.png,Lung_Opacity/masks/Lung_Opacity-4163.png,True,False,299,299,L,113.72,f422a120ab2d00ec5fc6322dda695bc5
Lung_Opacity-4164.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4164.png,Lung_Opacity/images/Lung_Opacity-4164.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4164.png,Lung_Opacity/masks/Lung_Opacity-4164.png,True,False,299,299,L,113.17,063fef49561f7cc796884380a8fd88d0
Lung_Opacity-4165.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4165.png,Lung_Opacity/images/Lung_Opacity-4165.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4165.png,Lung_Opacity/masks/Lung_Opacity-4165.png,True,False,299,299,L,99.15,5b191152d052e7f2c9a51e567babb148
Lung_Opacity-4166.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4166.png,Lung_Opacity/images/Lung_Opacity-4166.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4166.png,Lung_Opacity/masks/Lung_Opacity-4166.png,True,False,299,299,L,148.28,c35ab1b604f1c5d9092b60ce1eed7f61
Lung_Opacity-4167.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4167.png,Lung_Opacity/images/Lung_Opacity-4167.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4167.png,Lung_Opacity/masks/Lung_Opacity-4167.png,True,False,299,299,L,174.79,3592f74769d029c93672882b3dd0bad0
Lung_Opacity-4168.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4168.png,Lung_Opacity/images/Lung_Opacity-4168.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4168.png,Lung_Opacity/masks/Lung_Opacity-4168.png,True,False,299,299,L,119.25,3b3a789d3a7ade414f29ab3e9cf64003
Lung_Opacity-4169.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4169.png,Lung_Opacity/images/Lung_Opacity-4169.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4169.png,Lung_Opacity/masks/Lung_Opacity-4169.png,True,False,299,299,L,122.11,7f5867bf5285216dfb6388ef3309d2fd
Lung_Opacity-417.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-417.png,Lung_Opacity/images/Lung_Opacity-417.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-417.png,Lung_Opacity/masks/Lung_Opacity-417.png,True,False,299,299,L,108.72,41c5071e2df83ca3231354ca85a4841c
Lung_Opacity-4170.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4170.png,Lung_Opacity/images/Lung_Opacity-4170.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4170.png,Lung_Opacity/masks/Lung_Opacity-4170.png,True,False,299,299,L,150.8,b1cf8be48063a6e0cc2eba7d77de00f8
Lung_Opacity-4171.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4171.png,Lung_Opacity/images/Lung_Opacity-4171.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4171.png,Lung_Opacity/masks/Lung_Opacity-4171.png,True,False,299,299,L,115.09,db029936b4e03fc4bccec894340985d0
Lung_Opacity-4172.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4172.png,Lung_Opacity/images/Lung_Opacity-4172.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4172.png,Lung_Opacity/masks/Lung_Opacity-4172.png,True,False,299,299,L,119.08,e625a9917495574dbae3bf1f385e8f9f
Lung_Opacity-4173.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4173.png,Lung_Opacity/images/Lung_Opacity-4173.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4173.png,Lung_Opacity/masks/Lung_Opacity-4173.png,True,False,299,299,L,113.83,9624e593d651034963720268c8c1e58f
Lung_Opacity-4174.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4174.png,Lung_Opacity/images/Lung_Opacity-4174.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4174.png,Lung_Opacity/masks/Lung_Opacity-4174.png,True,False,299,299,L,111.7,ec12810992cbd106bcd37e58ddeab304
Lung_Opacity-4175.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4175.png,Lung_Opacity/images/Lung_Opacity-4175.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4175.png,Lung_Opacity/masks/Lung_Opacity-4175.png,True,False,299,299,L,99.19,5a0003f7937d0e39b0797c5ad34c1897
Lung_Opacity-4176.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4176.png,Lung_Opacity/images/Lung_Opacity-4176.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4176.png,Lung_Opacity/masks/Lung_Opacity-4176.png,True,False,299,299,L,125.91,54de35cac34b3c17c777b21bfbcdbbbd
Lung_Opacity-4177.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4177.png,Lung_Opacity/images/Lung_Opacity-4177.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4177.png,Lung_Opacity/masks/Lung_Opacity-4177.png,True,False,299,299,L,105.5,24fcbcb01bbe33ca2eba0bedf8ef7051
Lung_Opacity-4178.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4178.png,Lung_Opacity/images/Lung_Opacity-4178.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4178.png,Lung_Opacity/masks/Lung_Opacity-4178.png,True,False,299,299,L,130.16,5d0e42271f632582f323eadd28a0fb8b
Lung_Opacity-4179.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4179.png,Lung_Opacity/images/Lung_Opacity-4179.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4179.png,Lung_Opacity/masks/Lung_Opacity-4179.png,True,False,299,299,L,131.25,3e00ad090b2a2f78a54a1333ae4aa5b1
Lung_Opacity-418.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-418.png,Lung_Opacity/images/Lung_Opacity-418.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-418.png,Lung_Opacity/masks/Lung_Opacity-418.png,True,False,299,299,L,162.13,09a2cfdb2e76dd26f2d911391fcdaa5b
Lung_Opacity-4180.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4180.png,Lung_Opacity/images/Lung_Opacity-4180.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4180.png,Lung_Opacity/masks/Lung_Opacity-4180.png,True,False,299,299,L,107.39,fab57c4b4cfd89c2be5441204a537326
Lung_Opacity-4181.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4181.png,Lung_Opacity/images/Lung_Opacity-4181.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4181.png,Lung_Opacity/masks/Lung_Opacity-4181.png,True,False,299,299,L,139.33,4c6090a56751818fbfac7e013ee994f1
Lung_Opacity-4182.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4182.png,Lung_Opacity/images/Lung_Opacity-4182.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4182.png,Lung_Opacity/masks/Lung_Opacity-4182.png,True,False,299,299,L,159.55,c77dd1db9e167873501ca93de91dfba0
Lung_Opacity-4183.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4183.png,Lung_Opacity/images/Lung_Opacity-4183.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4183.png,Lung_Opacity/masks/Lung_Opacity-4183.png,True,False,299,299,L,124.42,adb87bc5606d20b90e93a9a7a5076f1a
Lung_Opacity-4184.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4184.png,Lung_Opacity/images/Lung_Opacity-4184.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4184.png,Lung_Opacity/masks/Lung_Opacity-4184.png,True,False,299,299,L,115.04,1e834b5ea9352dbd4eec3f06167eea7f
Lung_Opacity-4185.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4185.png,Lung_Opacity/images/Lung_Opacity-4185.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4185.png,Lung_Opacity/masks/Lung_Opacity-4185.png,True,False,299,299,L,125.53,484b43cd3189feea839b4b2617dec394
Lung_Opacity-4186.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4186.png,Lung_Opacity/images/Lung_Opacity-4186.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4186.png,Lung_Opacity/masks/Lung_Opacity-4186.png,True,False,299,299,L,116.49,b5a2e7176b364b8d40a21c8c1d864c67
Lung_Opacity-4187.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4187.png,Lung_Opacity/images/Lung_Opacity-4187.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4187.png,Lung_Opacity/masks/Lung_Opacity-4187.png,True,False,299,299,L,116.88,1dc1cd7ea59aa646bea1dfe7d020aa7a
Lung_Opacity-4188.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4188.png,Lung_Opacity/images/Lung_Opacity-4188.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4188.png,Lung_Opacity/masks/Lung_Opacity-4188.png,True,False,299,299,L,115.58,b35d045bb031bacdd2402847537d14cc
Lung_Opacity-4189.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4189.png,Lung_Opacity/images/Lung_Opacity-4189.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4189.png,Lung_Opacity/masks/Lung_Opacity-4189.png,True,False,299,299,L,137.11,751d21ba32d276dddbf89929708b7f53
Lung_Opacity-419.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-419.png,Lung_Opacity/images/Lung_Opacity-419.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-419.png,Lung_Opacity/masks/Lung_Opacity-419.png,True,False,299,299,L,120.64,c7c2851c3024767aa8c01c0c4129371f
Lung_Opacity-4190.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4190.png,Lung_Opacity/images/Lung_Opacity-4190.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4190.png,Lung_Opacity/masks/Lung_Opacity-4190.png,True,False,299,299,L,161.91,a73c350142a18d697e5a2fa410b79671
Lung_Opacity-4191.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4191.png,Lung_Opacity/images/Lung_Opacity-4191.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4191.png,Lung_Opacity/masks/Lung_Opacity-4191.png,True,False,299,299,L,163.91,1a7b2abcc719bdf6a8af08fbede9cc12
Lung_Opacity-4192.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4192.png,Lung_Opacity/images/Lung_Opacity-4192.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4192.png,Lung_Opacity/masks/Lung_Opacity-4192.png,True,False,299,299,L,164.9,21f0e352bb395d8a09127cdf85464940
Lung_Opacity-4193.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4193.png,Lung_Opacity/images/Lung_Opacity-4193.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4193.png,Lung_Opacity/masks/Lung_Opacity-4193.png,True,False,299,299,L,121.83,b7a35173ab1bb4c990e20ef1483af36c
Lung_Opacity-4194.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4194.png,Lung_Opacity/images/Lung_Opacity-4194.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4194.png,Lung_Opacity/masks/Lung_Opacity-4194.png,True,False,299,299,L,119.25,6cbca75dd013f8386a1963d3786bd5d2
Lung_Opacity-4195.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4195.png,Lung_Opacity/images/Lung_Opacity-4195.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4195.png,Lung_Opacity/masks/Lung_Opacity-4195.png,True,False,299,299,L,141.94,5759dc1dbe683f954a714a1d4abc54d5
Lung_Opacity-4196.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4196.png,Lung_Opacity/images/Lung_Opacity-4196.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4196.png,Lung_Opacity/masks/Lung_Opacity-4196.png,True,False,299,299,L,100.8,70423c647f10ac913c9a1d5cc16c4a5c
Lung_Opacity-4197.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4197.png,Lung_Opacity/images/Lung_Opacity-4197.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4197.png,Lung_Opacity/masks/Lung_Opacity-4197.png,True,False,299,299,L,166.79,77ef4af6fdc4e59e35ddcfe3e04a3f7e
Lung_Opacity-4198.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4198.png,Lung_Opacity/images/Lung_Opacity-4198.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4198.png,Lung_Opacity/masks/Lung_Opacity-4198.png,True,False,299,299,L,129.73,3b52e1b958c39b87fc74dbd3c7934ed0
Lung_Opacity-4199.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4199.png,Lung_Opacity/images/Lung_Opacity-4199.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4199.png,Lung_Opacity/masks/Lung_Opacity-4199.png,True,False,299,299,L,117.21,acd88d044aadaf5f2a3f8ff8021cd54a
Lung_Opacity-42.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-42.png,Lung_Opacity/images/Lung_Opacity-42.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-42.png,Lung_Opacity/masks/Lung_Opacity-42.png,True,False,299,299,L,118.92,035ab9175a8fb5a7e53f9ccda6854e86
Lung_Opacity-420.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-420.png,Lung_Opacity/images/Lung_Opacity-420.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-420.png,Lung_Opacity/masks/Lung_Opacity-420.png,True,False,299,299,L,104.56,6e47216733485a1d99b28e895087602d
Lung_Opacity-4200.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4200.png,Lung_Opacity/images/Lung_Opacity-4200.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4200.png,Lung_Opacity/masks/Lung_Opacity-4200.png,True,False,299,299,L,129.4,33a2be2b9751104baf45866065b27cf3
Lung_Opacity-4201.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4201.png,Lung_Opacity/images/Lung_Opacity-4201.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4201.png,Lung_Opacity/masks/Lung_Opacity-4201.png,True,False,299,299,L,136.14,a9800a270236fb7b26e7ff81de9291ee
Lung_Opacity-4202.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4202.png,Lung_Opacity/images/Lung_Opacity-4202.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4202.png,Lung_Opacity/masks/Lung_Opacity-4202.png,True,False,299,299,L,116.92,598796811ff2942601a28f2db4a67d73
Lung_Opacity-4203.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4203.png,Lung_Opacity/images/Lung_Opacity-4203.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4203.png,Lung_Opacity/masks/Lung_Opacity-4203.png,True,False,299,299,L,122.82,82ff9c3c992dc6c176cd275034f85f82
Lung_Opacity-4204.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4204.png,Lung_Opacity/images/Lung_Opacity-4204.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4204.png,Lung_Opacity/masks/Lung_Opacity-4204.png,True,False,299,299,L,164.21,dc4ae060a063385412902e0ca9c8a35b
Lung_Opacity-4205.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4205.png,Lung_Opacity/images/Lung_Opacity-4205.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4205.png,Lung_Opacity/masks/Lung_Opacity-4205.png,True,False,299,299,L,128.32,a58cce30f7d2ee558c70a593b59fb86c
Lung_Opacity-4206.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4206.png,Lung_Opacity/images/Lung_Opacity-4206.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4206.png,Lung_Opacity/masks/Lung_Opacity-4206.png,True,False,299,299,L,168.34,9aee63acee9def194b4caa7775769886
Lung_Opacity-4207.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4207.png,Lung_Opacity/images/Lung_Opacity-4207.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4207.png,Lung_Opacity/masks/Lung_Opacity-4207.png,True,False,299,299,L,112.07,104729964bcccd0f4f402853786a14eb
Lung_Opacity-4208.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4208.png,Lung_Opacity/images/Lung_Opacity-4208.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4208.png,Lung_Opacity/masks/Lung_Opacity-4208.png,True,False,299,299,L,113.18,4b9e5ce7a2d7b32b01a7905889ac3ad3
Lung_Opacity-4209.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4209.png,Lung_Opacity/images/Lung_Opacity-4209.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4209.png,Lung_Opacity/masks/Lung_Opacity-4209.png,True,False,299,299,L,144.51,907a27fcff60907a85a6893623af3819
Lung_Opacity-421.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-421.png,Lung_Opacity/images/Lung_Opacity-421.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-421.png,Lung_Opacity/masks/Lung_Opacity-421.png,True,False,299,299,L,100.98,4f2103773cb7a65d694fb3052355a2c9
Lung_Opacity-4210.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4210.png,Lung_Opacity/images/Lung_Opacity-4210.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4210.png,Lung_Opacity/masks/Lung_Opacity-4210.png,True,False,299,299,L,174.04,05bf4bec088c32e20664f1e4737b2ca5
Lung_Opacity-4211.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4211.png,Lung_Opacity/images/Lung_Opacity-4211.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4211.png,Lung_Opacity/masks/Lung_Opacity-4211.png,True,False,299,299,L,131.59,7293f564e67d57d9c679aa07ddd43dc2
Lung_Opacity-4212.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4212.png,Lung_Opacity/images/Lung_Opacity-4212.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4212.png,Lung_Opacity/masks/Lung_Opacity-4212.png,True,False,299,299,L,166.53,2556837d3266d427483f3222110d176d
Lung_Opacity-4213.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4213.png,Lung_Opacity/images/Lung_Opacity-4213.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4213.png,Lung_Opacity/masks/Lung_Opacity-4213.png,True,False,299,299,L,110.83,aa3a1628eb33ba60aa21e98b5561c5bc
Lung_Opacity-4214.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4214.png,Lung_Opacity/images/Lung_Opacity-4214.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4214.png,Lung_Opacity/masks/Lung_Opacity-4214.png,True,False,299,299,L,103.82,2f228ef70fc5adae0a09fdd5f5bd3f4c
Lung_Opacity-4215.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4215.png,Lung_Opacity/images/Lung_Opacity-4215.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4215.png,Lung_Opacity/masks/Lung_Opacity-4215.png,True,False,299,299,L,116.21,41ae12ebd5fd01fcc1b2ec14d5e47b98
Lung_Opacity-4216.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4216.png,Lung_Opacity/images/Lung_Opacity-4216.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4216.png,Lung_Opacity/masks/Lung_Opacity-4216.png,True,False,299,299,L,127.65,74c9e6bacc462e15f7c333f1ef63d109
Lung_Opacity-4217.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4217.png,Lung_Opacity/images/Lung_Opacity-4217.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4217.png,Lung_Opacity/masks/Lung_Opacity-4217.png,True,False,299,299,L,101.11,67829f6e22fcd06ec9179e4c1eda7343
Lung_Opacity-4218.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4218.png,Lung_Opacity/images/Lung_Opacity-4218.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4218.png,Lung_Opacity/masks/Lung_Opacity-4218.png,True,False,299,299,L,106.29,3c5163496b4edd648b3e69eeb7683aeb
Lung_Opacity-4219.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4219.png,Lung_Opacity/images/Lung_Opacity-4219.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4219.png,Lung_Opacity/masks/Lung_Opacity-4219.png,True,False,299,299,L,100.67,802d9a2d0f394e6e60b65d4b7258e85f
Lung_Opacity-422.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-422.png,Lung_Opacity/images/Lung_Opacity-422.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-422.png,Lung_Opacity/masks/Lung_Opacity-422.png,True,False,299,299,L,113.75,67cae12ba0536068fac1a59c2ef2b0f6
Lung_Opacity-4220.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4220.png,Lung_Opacity/images/Lung_Opacity-4220.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4220.png,Lung_Opacity/masks/Lung_Opacity-4220.png,True,False,299,299,L,111.03,012a7b48a08f045e1af94c22bc8373b9
Lung_Opacity-4221.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4221.png,Lung_Opacity/images/Lung_Opacity-4221.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4221.png,Lung_Opacity/masks/Lung_Opacity-4221.png,True,False,299,299,L,110.89,2f9e7a425216a74d964919a5d9bbfd1d
Lung_Opacity-4222.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4222.png,Lung_Opacity/images/Lung_Opacity-4222.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4222.png,Lung_Opacity/masks/Lung_Opacity-4222.png,True,False,299,299,L,142.11,7e20e55bb67ecd012699ba600e2cde19
Lung_Opacity-4223.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4223.png,Lung_Opacity/images/Lung_Opacity-4223.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4223.png,Lung_Opacity/masks/Lung_Opacity-4223.png,True,False,299,299,L,116.64,31c8cdb839dad6de514d0bc99e5c6016
Lung_Opacity-4224.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4224.png,Lung_Opacity/images/Lung_Opacity-4224.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4224.png,Lung_Opacity/masks/Lung_Opacity-4224.png,True,False,299,299,L,130.82,4a0168c56ae7498539deae24a40c38ca
Lung_Opacity-4225.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4225.png,Lung_Opacity/images/Lung_Opacity-4225.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4225.png,Lung_Opacity/masks/Lung_Opacity-4225.png,True,False,299,299,L,114.61,de82ebc91fdbc195fbe7573c4e75d410
Lung_Opacity-4226.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4226.png,Lung_Opacity/images/Lung_Opacity-4226.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4226.png,Lung_Opacity/masks/Lung_Opacity-4226.png,True,False,299,299,L,107.98,aeba042db929cdf0184b853a5eacfed2
Lung_Opacity-4227.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4227.png,Lung_Opacity/images/Lung_Opacity-4227.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4227.png,Lung_Opacity/masks/Lung_Opacity-4227.png,True,False,299,299,L,107.47,15336d105d8e4cbf7b3cbe4b012a6367
Lung_Opacity-4228.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4228.png,Lung_Opacity/images/Lung_Opacity-4228.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4228.png,Lung_Opacity/masks/Lung_Opacity-4228.png,True,False,299,299,L,133.9,4434637acd3a7e14d474c6df16a090f0
Lung_Opacity-4229.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4229.png,Lung_Opacity/images/Lung_Opacity-4229.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4229.png,Lung_Opacity/masks/Lung_Opacity-4229.png,True,False,299,299,L,163.12,d552d0274cdd3aa346a87aabeb484b0e
Lung_Opacity-423.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-423.png,Lung_Opacity/images/Lung_Opacity-423.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-423.png,Lung_Opacity/masks/Lung_Opacity-423.png,True,False,299,299,L,138.99,b72ff54ad7c8e8bb5b848a427c679226
Lung_Opacity-4230.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4230.png,Lung_Opacity/images/Lung_Opacity-4230.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4230.png,Lung_Opacity/masks/Lung_Opacity-4230.png,True,False,299,299,L,161.69,aa131113e79afe82334b4901074a4b7f
Lung_Opacity-4231.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4231.png,Lung_Opacity/images/Lung_Opacity-4231.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4231.png,Lung_Opacity/masks/Lung_Opacity-4231.png,True,False,299,299,L,151.05,42ea63563f3eeadac7ed94d5b0bee816
Lung_Opacity-4232.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4232.png,Lung_Opacity/images/Lung_Opacity-4232.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4232.png,Lung_Opacity/masks/Lung_Opacity-4232.png,True,False,299,299,L,125.34,78a9307b4f7ab2880235915db662bc11
Lung_Opacity-4233.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4233.png,Lung_Opacity/images/Lung_Opacity-4233.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4233.png,Lung_Opacity/masks/Lung_Opacity-4233.png,True,False,299,299,L,116.85,ae123ecaa7c113b3b3bc79f7714496e6
Lung_Opacity-4234.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4234.png,Lung_Opacity/images/Lung_Opacity-4234.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4234.png,Lung_Opacity/masks/Lung_Opacity-4234.png,True,False,299,299,L,160.0,5f2eb3db9effd5941cf1f340a6e9637f
Lung_Opacity-4235.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4235.png,Lung_Opacity/images/Lung_Opacity-4235.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4235.png,Lung_Opacity/masks/Lung_Opacity-4235.png,True,False,299,299,L,170.75,b3a9a8090814f8cf947f782c8b68748a
Lung_Opacity-4236.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4236.png,Lung_Opacity/images/Lung_Opacity-4236.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4236.png,Lung_Opacity/masks/Lung_Opacity-4236.png,True,False,299,299,L,145.77,c317e58fef8e6df773167cb549c86e7c
Lung_Opacity-4237.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4237.png,Lung_Opacity/images/Lung_Opacity-4237.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4237.png,Lung_Opacity/masks/Lung_Opacity-4237.png,True,False,299,299,L,128.02,98ea55eeb3b3f8842e0902480007ea57
Lung_Opacity-4238.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4238.png,Lung_Opacity/images/Lung_Opacity-4238.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4238.png,Lung_Opacity/masks/Lung_Opacity-4238.png,True,False,299,299,L,111.86,90ffd2ecd63f29aa022e67a6bdba3273
Lung_Opacity-4239.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4239.png,Lung_Opacity/images/Lung_Opacity-4239.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4239.png,Lung_Opacity/masks/Lung_Opacity-4239.png,True,False,299,299,L,148.0,3bf03eef66747845cfcc3df6011ef3f8
Lung_Opacity-424.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-424.png,Lung_Opacity/images/Lung_Opacity-424.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-424.png,Lung_Opacity/masks/Lung_Opacity-424.png,True,False,299,299,L,109.88,0b62a8db401ab442de8a15d1334495c3
Lung_Opacity-4240.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4240.png,Lung_Opacity/images/Lung_Opacity-4240.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4240.png,Lung_Opacity/masks/Lung_Opacity-4240.png,True,False,299,299,L,143.14,0a4d2c818be7b64232ab358eb5ddfb1f
Lung_Opacity-4241.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4241.png,Lung_Opacity/images/Lung_Opacity-4241.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4241.png,Lung_Opacity/masks/Lung_Opacity-4241.png,True,False,299,299,L,110.87,9b6b18debccd073a82e4217c63bd863b
Lung_Opacity-4242.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4242.png,Lung_Opacity/images/Lung_Opacity-4242.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4242.png,Lung_Opacity/masks/Lung_Opacity-4242.png,True,False,299,299,L,158.37,ca4c8e7ff5c8c475e6046c5285c8f670
Lung_Opacity-4243.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4243.png,Lung_Opacity/images/Lung_Opacity-4243.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4243.png,Lung_Opacity/masks/Lung_Opacity-4243.png,True,False,299,299,L,112.69,350f2b8c8ea1cbbe40a2b4782d1926d9
Lung_Opacity-4244.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4244.png,Lung_Opacity/images/Lung_Opacity-4244.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4244.png,Lung_Opacity/masks/Lung_Opacity-4244.png,True,False,299,299,L,111.64,e68f41d37afbe8ef27826b698ecd0272
Lung_Opacity-4245.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4245.png,Lung_Opacity/images/Lung_Opacity-4245.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4245.png,Lung_Opacity/masks/Lung_Opacity-4245.png,True,False,299,299,L,104.49,307fce69ee712ca505ce1344282f163a
Lung_Opacity-4246.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4246.png,Lung_Opacity/images/Lung_Opacity-4246.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4246.png,Lung_Opacity/masks/Lung_Opacity-4246.png,True,False,299,299,L,131.28,671897cabb7aba584167cc953284c351
Lung_Opacity-4247.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4247.png,Lung_Opacity/images/Lung_Opacity-4247.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4247.png,Lung_Opacity/masks/Lung_Opacity-4247.png,True,False,299,299,L,102.82,50718fb5db34a85f1a8fa98cbe6b07ab
Lung_Opacity-4248.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4248.png,Lung_Opacity/images/Lung_Opacity-4248.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4248.png,Lung_Opacity/masks/Lung_Opacity-4248.png,True,False,299,299,L,152.99,cd2ab43e025feb74aa8b5eac14164c6a
Lung_Opacity-4249.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4249.png,Lung_Opacity/images/Lung_Opacity-4249.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4249.png,Lung_Opacity/masks/Lung_Opacity-4249.png,True,False,299,299,L,131.25,73853060ae10158e0ab5b5c9d029000e
Lung_Opacity-425.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-425.png,Lung_Opacity/images/Lung_Opacity-425.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-425.png,Lung_Opacity/masks/Lung_Opacity-425.png,True,False,299,299,L,121.52,e4212d7d64721c243dade70d62bf2cf6
Lung_Opacity-4250.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4250.png,Lung_Opacity/images/Lung_Opacity-4250.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4250.png,Lung_Opacity/masks/Lung_Opacity-4250.png,True,False,299,299,L,107.44,a2ffa61117cc9920ed68f3d2b6ddfbc9
Lung_Opacity-4251.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4251.png,Lung_Opacity/images/Lung_Opacity-4251.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4251.png,Lung_Opacity/masks/Lung_Opacity-4251.png,True,False,299,299,L,157.44,d8c902187ebbcf474af86eb0bf4877a8
Lung_Opacity-4252.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4252.png,Lung_Opacity/images/Lung_Opacity-4252.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4252.png,Lung_Opacity/masks/Lung_Opacity-4252.png,True,False,299,299,L,128.03,20f16880a40e77dd543347e3d0ebba36
Lung_Opacity-4253.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4253.png,Lung_Opacity/images/Lung_Opacity-4253.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4253.png,Lung_Opacity/masks/Lung_Opacity-4253.png,True,False,299,299,L,115.25,295a7e93e5b664730bd0447d4e93b62c
Lung_Opacity-4254.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4254.png,Lung_Opacity/images/Lung_Opacity-4254.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4254.png,Lung_Opacity/masks/Lung_Opacity-4254.png,True,False,299,299,L,103.33,9131caf2bd1c22b6b3488dd440283a57
Lung_Opacity-4255.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4255.png,Lung_Opacity/images/Lung_Opacity-4255.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4255.png,Lung_Opacity/masks/Lung_Opacity-4255.png,True,False,299,299,L,157.45,fae9416bc2ff0740f0c14b97ce2787f6
Lung_Opacity-4256.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4256.png,Lung_Opacity/images/Lung_Opacity-4256.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4256.png,Lung_Opacity/masks/Lung_Opacity-4256.png,True,False,299,299,L,125.99,a2d79d46af001d03e0e6f316dd3e1ec7
Lung_Opacity-4257.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4257.png,Lung_Opacity/images/Lung_Opacity-4257.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4257.png,Lung_Opacity/masks/Lung_Opacity-4257.png,True,False,299,299,L,169.83,44656f8c2acae50d2237b21855900158
Lung_Opacity-4258.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4258.png,Lung_Opacity/images/Lung_Opacity-4258.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4258.png,Lung_Opacity/masks/Lung_Opacity-4258.png,True,False,299,299,L,155.13,d50837d003e126a4ac88073d88a15e74
Lung_Opacity-4259.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4259.png,Lung_Opacity/images/Lung_Opacity-4259.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4259.png,Lung_Opacity/masks/Lung_Opacity-4259.png,True,False,299,299,L,152.14,8e6c679854095840e4e08d01b4aad02b
Lung_Opacity-426.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-426.png,Lung_Opacity/images/Lung_Opacity-426.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-426.png,Lung_Opacity/masks/Lung_Opacity-426.png,True,False,299,299,L,162.3,f8b5889f5270247093a49f642a744877
Lung_Opacity-4260.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4260.png,Lung_Opacity/images/Lung_Opacity-4260.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4260.png,Lung_Opacity/masks/Lung_Opacity-4260.png,True,False,299,299,L,120.09,0c4a53816e38b1dc02119e1a71a1f30c
Lung_Opacity-4261.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4261.png,Lung_Opacity/images/Lung_Opacity-4261.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4261.png,Lung_Opacity/masks/Lung_Opacity-4261.png,True,False,299,299,L,90.71,90346a5be76a31b86701bb27533d14d1
Lung_Opacity-4262.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4262.png,Lung_Opacity/images/Lung_Opacity-4262.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4262.png,Lung_Opacity/masks/Lung_Opacity-4262.png,True,False,299,299,L,160.71,a2e85b1b51d78718580f9586467fe97a
Lung_Opacity-4263.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4263.png,Lung_Opacity/images/Lung_Opacity-4263.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4263.png,Lung_Opacity/masks/Lung_Opacity-4263.png,True,False,299,299,L,128.02,bb525ea8d114ec7c60f84107033d5b36
Lung_Opacity-4264.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4264.png,Lung_Opacity/images/Lung_Opacity-4264.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4264.png,Lung_Opacity/masks/Lung_Opacity-4264.png,True,False,299,299,L,160.6,60940f1995c1fa577c68de7c838b882e
Lung_Opacity-4265.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4265.png,Lung_Opacity/images/Lung_Opacity-4265.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4265.png,Lung_Opacity/masks/Lung_Opacity-4265.png,True,False,299,299,L,110.66,8dbf221ee4fbd7d2f64d12d98e4e5bf5
Lung_Opacity-4266.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4266.png,Lung_Opacity/images/Lung_Opacity-4266.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4266.png,Lung_Opacity/masks/Lung_Opacity-4266.png,True,False,299,299,L,123.68,870c06f8966470e92ce5784d407bce79
Lung_Opacity-4267.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4267.png,Lung_Opacity/images/Lung_Opacity-4267.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4267.png,Lung_Opacity/masks/Lung_Opacity-4267.png,True,False,299,299,L,109.8,0bbe61751619f907c4ce75e21e50edd2
Lung_Opacity-4268.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4268.png,Lung_Opacity/images/Lung_Opacity-4268.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4268.png,Lung_Opacity/masks/Lung_Opacity-4268.png,True,False,299,299,L,123.76,c5889c593640565b86bfea5d0469b6d7
Lung_Opacity-4269.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4269.png,Lung_Opacity/images/Lung_Opacity-4269.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4269.png,Lung_Opacity/masks/Lung_Opacity-4269.png,True,False,299,299,L,143.5,e48cd840b7e9ea09638d390c8d1c0ac1
Lung_Opacity-427.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-427.png,Lung_Opacity/images/Lung_Opacity-427.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-427.png,Lung_Opacity/masks/Lung_Opacity-427.png,True,False,299,299,L,151.4,4219282c277a0388d52f0ac7a741413b
Lung_Opacity-4270.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4270.png,Lung_Opacity/images/Lung_Opacity-4270.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4270.png,Lung_Opacity/masks/Lung_Opacity-4270.png,True,False,299,299,L,161.71,f8823e6e99844faf0deab9b9ed8fdace
Lung_Opacity-4271.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4271.png,Lung_Opacity/images/Lung_Opacity-4271.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4271.png,Lung_Opacity/masks/Lung_Opacity-4271.png,True,False,299,299,L,129.32,fb6ab467d37d1f650edbef53d8351321
Lung_Opacity-4272.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4272.png,Lung_Opacity/images/Lung_Opacity-4272.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4272.png,Lung_Opacity/masks/Lung_Opacity-4272.png,True,False,299,299,L,134.0,b23a3b3190d08a6e630f6116e25c46fb
Lung_Opacity-4273.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4273.png,Lung_Opacity/images/Lung_Opacity-4273.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4273.png,Lung_Opacity/masks/Lung_Opacity-4273.png,True,False,299,299,L,174.77,e59e6fa883f6bbc36283e89752fed34e
Lung_Opacity-4274.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4274.png,Lung_Opacity/images/Lung_Opacity-4274.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4274.png,Lung_Opacity/masks/Lung_Opacity-4274.png,True,False,299,299,L,116.21,f66ce3ea081656ac04723d95ace33159
Lung_Opacity-4275.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4275.png,Lung_Opacity/images/Lung_Opacity-4275.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4275.png,Lung_Opacity/masks/Lung_Opacity-4275.png,True,False,299,299,L,117.71,d3ad9a4cf630dbcb5cbd6982ca55bfbc
Lung_Opacity-4276.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4276.png,Lung_Opacity/images/Lung_Opacity-4276.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4276.png,Lung_Opacity/masks/Lung_Opacity-4276.png,True,False,299,299,L,104.85,82972d205d1c0286ef8a619695f3d1dd
Lung_Opacity-4277.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4277.png,Lung_Opacity/images/Lung_Opacity-4277.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4277.png,Lung_Opacity/masks/Lung_Opacity-4277.png,True,False,299,299,L,104.77,360f399709147c02f28663b77a97523a
Lung_Opacity-4278.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4278.png,Lung_Opacity/images/Lung_Opacity-4278.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4278.png,Lung_Opacity/masks/Lung_Opacity-4278.png,True,False,299,299,L,96.25,79996f3b96af07b39ba85ac25710a21c
Lung_Opacity-4279.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4279.png,Lung_Opacity/images/Lung_Opacity-4279.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4279.png,Lung_Opacity/masks/Lung_Opacity-4279.png,True,False,299,299,L,108.72,1d97918cf982e7a0bb445caad65cd51f
Lung_Opacity-428.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-428.png,Lung_Opacity/images/Lung_Opacity-428.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-428.png,Lung_Opacity/masks/Lung_Opacity-428.png,True,False,299,299,L,143.49,68c27a4ed72188d141c65d61979b48f2
Lung_Opacity-4280.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4280.png,Lung_Opacity/images/Lung_Opacity-4280.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4280.png,Lung_Opacity/masks/Lung_Opacity-4280.png,True,False,299,299,L,163.18,e1cfcf7625b3c55382a7dee4f9507d1e
Lung_Opacity-4281.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4281.png,Lung_Opacity/images/Lung_Opacity-4281.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4281.png,Lung_Opacity/masks/Lung_Opacity-4281.png,True,False,299,299,L,118.98,8e07c5423d501a7f8b7439093c323174
Lung_Opacity-4282.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4282.png,Lung_Opacity/images/Lung_Opacity-4282.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4282.png,Lung_Opacity/masks/Lung_Opacity-4282.png,True,False,299,299,L,148.98,d12bb6d61337c5e82c238029c0b6fe5f
Lung_Opacity-4283.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4283.png,Lung_Opacity/images/Lung_Opacity-4283.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4283.png,Lung_Opacity/masks/Lung_Opacity-4283.png,True,False,299,299,L,100.47,e40ff410be1fd6875a6fe89d1cb10075
Lung_Opacity-4284.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4284.png,Lung_Opacity/images/Lung_Opacity-4284.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4284.png,Lung_Opacity/masks/Lung_Opacity-4284.png,True,False,299,299,L,95.57,17ed5fb322083a99eb59fd01bf5b0e2c
Lung_Opacity-4285.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4285.png,Lung_Opacity/images/Lung_Opacity-4285.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4285.png,Lung_Opacity/masks/Lung_Opacity-4285.png,True,False,299,299,L,157.01,4a5be1b62c5620db89087ba5b8ccd15c
Lung_Opacity-4286.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4286.png,Lung_Opacity/images/Lung_Opacity-4286.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4286.png,Lung_Opacity/masks/Lung_Opacity-4286.png,True,False,299,299,L,98.52,6926ed84ef677ba542d4687ee70915d1
Lung_Opacity-4287.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4287.png,Lung_Opacity/images/Lung_Opacity-4287.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4287.png,Lung_Opacity/masks/Lung_Opacity-4287.png,True,False,299,299,L,94.01,e59285f6a45a1072a73c72fb102701b1
Lung_Opacity-4288.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4288.png,Lung_Opacity/images/Lung_Opacity-4288.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4288.png,Lung_Opacity/masks/Lung_Opacity-4288.png,True,False,299,299,L,166.19,d262698633e64613785216631f3907a1
Lung_Opacity-4289.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4289.png,Lung_Opacity/images/Lung_Opacity-4289.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4289.png,Lung_Opacity/masks/Lung_Opacity-4289.png,True,False,299,299,L,111.89,b37c57491dc3c54c49f224e0ed095ee4
Lung_Opacity-429.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-429.png,Lung_Opacity/images/Lung_Opacity-429.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-429.png,Lung_Opacity/masks/Lung_Opacity-429.png,True,False,299,299,L,83.98,2e153e00d7a195ae302811546e4bc285
Lung_Opacity-4290.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4290.png,Lung_Opacity/images/Lung_Opacity-4290.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4290.png,Lung_Opacity/masks/Lung_Opacity-4290.png,True,False,299,299,L,158.3,d9f2b1f2e9a9c4506f184e71c9f0c56d
Lung_Opacity-4291.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4291.png,Lung_Opacity/images/Lung_Opacity-4291.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4291.png,Lung_Opacity/masks/Lung_Opacity-4291.png,True,False,299,299,L,50.74,6f88041935d4d47a542ea9ae22b897f3
Lung_Opacity-4292.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4292.png,Lung_Opacity/images/Lung_Opacity-4292.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4292.png,Lung_Opacity/masks/Lung_Opacity-4292.png,True,False,299,299,L,114.98,b0536cb3c0af742c90e0314e824861bf
Lung_Opacity-4293.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4293.png,Lung_Opacity/images/Lung_Opacity-4293.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4293.png,Lung_Opacity/masks/Lung_Opacity-4293.png,True,False,299,299,L,142.12,b81f7263dd7cf971c4b9bd38c9af9886
Lung_Opacity-4294.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4294.png,Lung_Opacity/images/Lung_Opacity-4294.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4294.png,Lung_Opacity/masks/Lung_Opacity-4294.png,True,False,299,299,L,106.37,36d5bcbe64344da96ef339634ec44659
Lung_Opacity-4295.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4295.png,Lung_Opacity/images/Lung_Opacity-4295.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4295.png,Lung_Opacity/masks/Lung_Opacity-4295.png,True,False,299,299,L,170.35,f4322b638d144e960450eb175194d226
Lung_Opacity-4296.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4296.png,Lung_Opacity/images/Lung_Opacity-4296.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4296.png,Lung_Opacity/masks/Lung_Opacity-4296.png,True,False,299,299,L,118.87,7d5f26631e209b61aff19e05ccf2ee9e
Lung_Opacity-4297.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4297.png,Lung_Opacity/images/Lung_Opacity-4297.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4297.png,Lung_Opacity/masks/Lung_Opacity-4297.png,True,False,299,299,L,150.49,8bc66d52b57c8dd4bcb070a9f49529c8
Lung_Opacity-4298.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4298.png,Lung_Opacity/images/Lung_Opacity-4298.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4298.png,Lung_Opacity/masks/Lung_Opacity-4298.png,True,False,299,299,L,32.23,dc048c758067c3dae7ea3b7fda2c5fab
Lung_Opacity-4299.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4299.png,Lung_Opacity/images/Lung_Opacity-4299.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4299.png,Lung_Opacity/masks/Lung_Opacity-4299.png,True,False,299,299,L,111.77,0f55a1f55f64311843815192fbc0a73c
Lung_Opacity-43.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-43.png,Lung_Opacity/images/Lung_Opacity-43.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-43.png,Lung_Opacity/masks/Lung_Opacity-43.png,True,False,299,299,L,98.11,38b9d62e8c9a1625d9a90d5d2a4b70fb
Lung_Opacity-430.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-430.png,Lung_Opacity/images/Lung_Opacity-430.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-430.png,Lung_Opacity/masks/Lung_Opacity-430.png,True,False,299,299,L,133.91,41f66cb0b0ff4c020dec109328c34756
Lung_Opacity-4300.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4300.png,Lung_Opacity/images/Lung_Opacity-4300.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4300.png,Lung_Opacity/masks/Lung_Opacity-4300.png,True,False,299,299,L,115.76,5d7aaf983a0e116e4110ef930d867525
Lung_Opacity-4301.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4301.png,Lung_Opacity/images/Lung_Opacity-4301.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4301.png,Lung_Opacity/masks/Lung_Opacity-4301.png,True,False,299,299,L,148.68,c3699ab065176af98c3f40828ca0d7f9
Lung_Opacity-4302.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4302.png,Lung_Opacity/images/Lung_Opacity-4302.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4302.png,Lung_Opacity/masks/Lung_Opacity-4302.png,True,False,299,299,L,119.35,89834f0bc0a16ca30242c116bcdfe440
Lung_Opacity-4303.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4303.png,Lung_Opacity/images/Lung_Opacity-4303.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4303.png,Lung_Opacity/masks/Lung_Opacity-4303.png,True,False,299,299,L,161.5,094d71e46c03fd193eac4377ac5d53d9
Lung_Opacity-4304.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4304.png,Lung_Opacity/images/Lung_Opacity-4304.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4304.png,Lung_Opacity/masks/Lung_Opacity-4304.png,True,False,299,299,L,88.99,fe804d57c30579033d73fc3276767c07
Lung_Opacity-4305.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4305.png,Lung_Opacity/images/Lung_Opacity-4305.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4305.png,Lung_Opacity/masks/Lung_Opacity-4305.png,True,False,299,299,L,117.34,e821840af80af95b289a39ed6ff40c34
Lung_Opacity-4306.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4306.png,Lung_Opacity/images/Lung_Opacity-4306.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4306.png,Lung_Opacity/masks/Lung_Opacity-4306.png,True,False,299,299,L,142.91,80c354a185c6ebe047d3bba3278042a5
Lung_Opacity-4307.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4307.png,Lung_Opacity/images/Lung_Opacity-4307.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4307.png,Lung_Opacity/masks/Lung_Opacity-4307.png,True,False,299,299,L,146.12,4320fed614d019d3eeb1ddef45b3633c
Lung_Opacity-4308.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4308.png,Lung_Opacity/images/Lung_Opacity-4308.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4308.png,Lung_Opacity/masks/Lung_Opacity-4308.png,True,False,299,299,L,109.91,ffaea553933bf3f751b2703c9858a0d8
Lung_Opacity-4309.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4309.png,Lung_Opacity/images/Lung_Opacity-4309.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4309.png,Lung_Opacity/masks/Lung_Opacity-4309.png,True,False,299,299,L,158.18,67aca8d0f86a54c0fbf856b8b9c7147d
Lung_Opacity-431.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-431.png,Lung_Opacity/images/Lung_Opacity-431.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-431.png,Lung_Opacity/masks/Lung_Opacity-431.png,True,False,299,299,L,134.38,eed458fdfe0b192335121714019575bc
Lung_Opacity-4310.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4310.png,Lung_Opacity/images/Lung_Opacity-4310.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4310.png,Lung_Opacity/masks/Lung_Opacity-4310.png,True,False,299,299,L,149.12,b1d3ccbcd3d7f543abe61768120c63f3
Lung_Opacity-4311.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4311.png,Lung_Opacity/images/Lung_Opacity-4311.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4311.png,Lung_Opacity/masks/Lung_Opacity-4311.png,True,False,299,299,L,111.88,18460197d60aa838f918873f179ef97a
Lung_Opacity-4312.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4312.png,Lung_Opacity/images/Lung_Opacity-4312.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4312.png,Lung_Opacity/masks/Lung_Opacity-4312.png,True,False,299,299,L,106.29,1057a1c06fa8ca4eb1c9aaedf2bda378
Lung_Opacity-4313.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4313.png,Lung_Opacity/images/Lung_Opacity-4313.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4313.png,Lung_Opacity/masks/Lung_Opacity-4313.png,True,False,299,299,L,117.2,eb666073b0691c405dc27a85fbe29c6a
Lung_Opacity-4314.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4314.png,Lung_Opacity/images/Lung_Opacity-4314.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4314.png,Lung_Opacity/masks/Lung_Opacity-4314.png,True,False,299,299,L,63.88,d19d9fb926d9df5a114d999d695e812e
Lung_Opacity-4315.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4315.png,Lung_Opacity/images/Lung_Opacity-4315.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4315.png,Lung_Opacity/masks/Lung_Opacity-4315.png,True,False,299,299,L,111.36,268f1d885ae1186d45613df3ae6770e1
Lung_Opacity-4316.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4316.png,Lung_Opacity/images/Lung_Opacity-4316.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4316.png,Lung_Opacity/masks/Lung_Opacity-4316.png,True,False,299,299,L,103.76,ced636d51a10377f0f34188152724727
Lung_Opacity-4317.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4317.png,Lung_Opacity/images/Lung_Opacity-4317.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4317.png,Lung_Opacity/masks/Lung_Opacity-4317.png,True,False,299,299,L,127.6,57d0cf527d9a7009f9b190665968aab5
Lung_Opacity-4318.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4318.png,Lung_Opacity/images/Lung_Opacity-4318.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4318.png,Lung_Opacity/masks/Lung_Opacity-4318.png,True,False,299,299,L,177.91,cb0d372db143818458d13f7c64608f5c
Lung_Opacity-4319.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4319.png,Lung_Opacity/images/Lung_Opacity-4319.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4319.png,Lung_Opacity/masks/Lung_Opacity-4319.png,True,False,299,299,L,175.36,dc0d3dc39e13d0598cc09e0e7762aefc
Lung_Opacity-432.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-432.png,Lung_Opacity/images/Lung_Opacity-432.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-432.png,Lung_Opacity/masks/Lung_Opacity-432.png,True,False,299,299,L,98.78,dbc6181b7bccff28d43044611beb65dc
Lung_Opacity-4320.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4320.png,Lung_Opacity/images/Lung_Opacity-4320.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4320.png,Lung_Opacity/masks/Lung_Opacity-4320.png,True,False,299,299,L,93.53,246c9b28b26a3ca14c4700d148c5fb42
Lung_Opacity-4321.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4321.png,Lung_Opacity/images/Lung_Opacity-4321.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4321.png,Lung_Opacity/masks/Lung_Opacity-4321.png,True,False,299,299,L,142.11,e7efcf0d7bb1c713535a5844d60eab67
Lung_Opacity-4322.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4322.png,Lung_Opacity/images/Lung_Opacity-4322.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4322.png,Lung_Opacity/masks/Lung_Opacity-4322.png,True,False,299,299,L,87.57,7eb741706a5c21632ba40ae6a7253d30
Lung_Opacity-4323.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4323.png,Lung_Opacity/images/Lung_Opacity-4323.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4323.png,Lung_Opacity/masks/Lung_Opacity-4323.png,True,False,299,299,L,128.47,469cd1cbecf82e1e6787c14eb3ec72f8
Lung_Opacity-4324.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4324.png,Lung_Opacity/images/Lung_Opacity-4324.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4324.png,Lung_Opacity/masks/Lung_Opacity-4324.png,True,False,299,299,L,120.95,e913710052e9715eedcb84bf9757fa5a
Lung_Opacity-4325.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4325.png,Lung_Opacity/images/Lung_Opacity-4325.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4325.png,Lung_Opacity/masks/Lung_Opacity-4325.png,True,False,299,299,L,122.87,b638317f631cc758f19cd9a25f370fec
Lung_Opacity-4326.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4326.png,Lung_Opacity/images/Lung_Opacity-4326.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4326.png,Lung_Opacity/masks/Lung_Opacity-4326.png,True,False,299,299,L,147.38,dbd05f4dd0dbdb01346966f608dc0791
Lung_Opacity-4327.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4327.png,Lung_Opacity/images/Lung_Opacity-4327.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4327.png,Lung_Opacity/masks/Lung_Opacity-4327.png,True,False,299,299,L,58.03,bc5b32f5f1ef39fbceea25a816f9a972
Lung_Opacity-4328.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4328.png,Lung_Opacity/images/Lung_Opacity-4328.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4328.png,Lung_Opacity/masks/Lung_Opacity-4328.png,True,False,299,299,L,147.3,9d50cd0824d135d9765f32d1c53037d0
Lung_Opacity-4329.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4329.png,Lung_Opacity/images/Lung_Opacity-4329.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4329.png,Lung_Opacity/masks/Lung_Opacity-4329.png,True,False,299,299,L,114.59,3c73d169fa9a4343530f8259eb0cc777
Lung_Opacity-433.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-433.png,Lung_Opacity/images/Lung_Opacity-433.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-433.png,Lung_Opacity/masks/Lung_Opacity-433.png,True,False,299,299,L,139.1,04096d6cfa146e91330f9b43df8208a7
Lung_Opacity-4330.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4330.png,Lung_Opacity/images/Lung_Opacity-4330.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4330.png,Lung_Opacity/masks/Lung_Opacity-4330.png,True,False,299,299,L,108.43,d8bec4ba4c84dc09dce8a026baf3c17b
Lung_Opacity-4331.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4331.png,Lung_Opacity/images/Lung_Opacity-4331.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4331.png,Lung_Opacity/masks/Lung_Opacity-4331.png,True,False,299,299,L,120.81,a7a094f1412280ece6f89a8098a55b2c
Lung_Opacity-4332.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4332.png,Lung_Opacity/images/Lung_Opacity-4332.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4332.png,Lung_Opacity/masks/Lung_Opacity-4332.png,True,False,299,299,L,173.89,0f107c607eb4d634a9f00141996de640
Lung_Opacity-4333.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4333.png,Lung_Opacity/images/Lung_Opacity-4333.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4333.png,Lung_Opacity/masks/Lung_Opacity-4333.png,True,False,299,299,L,129.74,d33b52482aa9df048a489df0231376c4
Lung_Opacity-4334.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4334.png,Lung_Opacity/images/Lung_Opacity-4334.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4334.png,Lung_Opacity/masks/Lung_Opacity-4334.png,True,False,299,299,L,100.35,148ac5648f3ac6854c9f24d93fe05ee3
Lung_Opacity-4335.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4335.png,Lung_Opacity/images/Lung_Opacity-4335.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4335.png,Lung_Opacity/masks/Lung_Opacity-4335.png,True,False,299,299,L,148.02,2214798af4d991bdb8b8be0cda80eace
Lung_Opacity-4336.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4336.png,Lung_Opacity/images/Lung_Opacity-4336.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4336.png,Lung_Opacity/masks/Lung_Opacity-4336.png,True,False,299,299,L,110.01,69bc5b96b47a98d10eda82f9027df5a4
Lung_Opacity-4337.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4337.png,Lung_Opacity/images/Lung_Opacity-4337.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4337.png,Lung_Opacity/masks/Lung_Opacity-4337.png,True,False,299,299,L,124.94,08d3e285f316db04af0eca4354b05c5f
Lung_Opacity-4338.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4338.png,Lung_Opacity/images/Lung_Opacity-4338.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4338.png,Lung_Opacity/masks/Lung_Opacity-4338.png,True,False,299,299,L,137.06,7fa6d66f489ce7c6f7e7c0d29415e3a6
Lung_Opacity-4339.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4339.png,Lung_Opacity/images/Lung_Opacity-4339.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4339.png,Lung_Opacity/masks/Lung_Opacity-4339.png,True,False,299,299,L,127.87,b79601e3e0621d4c2e5b3be57910b657
Lung_Opacity-434.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-434.png,Lung_Opacity/images/Lung_Opacity-434.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-434.png,Lung_Opacity/masks/Lung_Opacity-434.png,True,False,299,299,L,98.45,8459af43f5407d9ad33a40274c92d816
Lung_Opacity-4340.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4340.png,Lung_Opacity/images/Lung_Opacity-4340.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4340.png,Lung_Opacity/masks/Lung_Opacity-4340.png,True,False,299,299,L,137.35,7641854e2d44962240ca711e11f3f7ac
Lung_Opacity-4341.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4341.png,Lung_Opacity/images/Lung_Opacity-4341.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4341.png,Lung_Opacity/masks/Lung_Opacity-4341.png,True,False,299,299,L,111.49,b279df8d62ddd3394c35df17467ea7df
Lung_Opacity-4342.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4342.png,Lung_Opacity/images/Lung_Opacity-4342.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4342.png,Lung_Opacity/masks/Lung_Opacity-4342.png,True,False,299,299,L,110.73,de84872c50acf127842ad5ece3acd5e9
Lung_Opacity-4343.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4343.png,Lung_Opacity/images/Lung_Opacity-4343.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4343.png,Lung_Opacity/masks/Lung_Opacity-4343.png,True,False,299,299,L,127.43,dadd3cc8d7f2000c5cc0df934a6d0dad
Lung_Opacity-4344.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4344.png,Lung_Opacity/images/Lung_Opacity-4344.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4344.png,Lung_Opacity/masks/Lung_Opacity-4344.png,True,False,299,299,L,107.13,2bd85eaa42869bc6e14555665ff998cf
Lung_Opacity-4345.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4345.png,Lung_Opacity/images/Lung_Opacity-4345.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4345.png,Lung_Opacity/masks/Lung_Opacity-4345.png,True,False,299,299,L,114.34,87ca1fa166019feb45b91cf9ab6b143f
Lung_Opacity-4346.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4346.png,Lung_Opacity/images/Lung_Opacity-4346.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4346.png,Lung_Opacity/masks/Lung_Opacity-4346.png,True,False,299,299,L,148.83,ddf06c727cdace13b23063420cad3b3e
Lung_Opacity-4347.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4347.png,Lung_Opacity/images/Lung_Opacity-4347.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4347.png,Lung_Opacity/masks/Lung_Opacity-4347.png,True,False,299,299,L,162.54,5191411835a24e193157c44e6a89d026
Lung_Opacity-4348.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4348.png,Lung_Opacity/images/Lung_Opacity-4348.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4348.png,Lung_Opacity/masks/Lung_Opacity-4348.png,True,False,299,299,L,155.58,47faaa5b10c984775ce307867cb5faba
Lung_Opacity-4349.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4349.png,Lung_Opacity/images/Lung_Opacity-4349.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4349.png,Lung_Opacity/masks/Lung_Opacity-4349.png,True,False,299,299,L,117.32,011e0a87a0831acc0d928fa63d7c3069
Lung_Opacity-435.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-435.png,Lung_Opacity/images/Lung_Opacity-435.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-435.png,Lung_Opacity/masks/Lung_Opacity-435.png,True,False,299,299,L,135.85,7e4a4f862d175865d2deb6e7ebaefa4c
Lung_Opacity-4350.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4350.png,Lung_Opacity/images/Lung_Opacity-4350.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4350.png,Lung_Opacity/masks/Lung_Opacity-4350.png,True,False,299,299,L,170.67,83793f1f7420d89c80fba148d04c0c71
Lung_Opacity-4351.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4351.png,Lung_Opacity/images/Lung_Opacity-4351.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4351.png,Lung_Opacity/masks/Lung_Opacity-4351.png,True,False,299,299,L,137.36,1af61c6ce5383dea5e1b209c706222bd
Lung_Opacity-4352.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4352.png,Lung_Opacity/images/Lung_Opacity-4352.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4352.png,Lung_Opacity/masks/Lung_Opacity-4352.png,True,False,299,299,L,135.52,f76c3c0fcadd02158dca69280976db25
Lung_Opacity-4353.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4353.png,Lung_Opacity/images/Lung_Opacity-4353.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4353.png,Lung_Opacity/masks/Lung_Opacity-4353.png,True,False,299,299,L,158.98,156b9b123fabec8cb9b8abefd97bb255
Lung_Opacity-4354.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4354.png,Lung_Opacity/images/Lung_Opacity-4354.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4354.png,Lung_Opacity/masks/Lung_Opacity-4354.png,True,False,299,299,L,107.07,a888a08139290fa4a62c8e664dd2e253
Lung_Opacity-4355.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4355.png,Lung_Opacity/images/Lung_Opacity-4355.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4355.png,Lung_Opacity/masks/Lung_Opacity-4355.png,True,False,299,299,L,166.07,825d3721886b91baa39c5ae2f87deaf2
Lung_Opacity-4356.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4356.png,Lung_Opacity/images/Lung_Opacity-4356.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4356.png,Lung_Opacity/masks/Lung_Opacity-4356.png,True,False,299,299,L,139.01,7e348a8a1e7f44bbad16cb3eb8dc5aea
Lung_Opacity-4357.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4357.png,Lung_Opacity/images/Lung_Opacity-4357.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4357.png,Lung_Opacity/masks/Lung_Opacity-4357.png,True,False,299,299,L,119.49,a7e8cf03e1f186c7ba8ee9fe9e2e3711
Lung_Opacity-4358.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4358.png,Lung_Opacity/images/Lung_Opacity-4358.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4358.png,Lung_Opacity/masks/Lung_Opacity-4358.png,True,False,299,299,L,115.03,1df84b99c0bdb6b926b59465ee8dadba
Lung_Opacity-4359.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4359.png,Lung_Opacity/images/Lung_Opacity-4359.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4359.png,Lung_Opacity/masks/Lung_Opacity-4359.png,True,False,299,299,L,121.87,b1cf7ca427ee128d3e6157c8e8220f78
Lung_Opacity-436.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-436.png,Lung_Opacity/images/Lung_Opacity-436.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-436.png,Lung_Opacity/masks/Lung_Opacity-436.png,True,False,299,299,L,111.07,0db6599f74d92be9104e20fc9dca51e0
Lung_Opacity-4360.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4360.png,Lung_Opacity/images/Lung_Opacity-4360.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4360.png,Lung_Opacity/masks/Lung_Opacity-4360.png,True,False,299,299,L,177.39,ea3d1dc71904753eebd8693e4be81c45
Lung_Opacity-4361.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4361.png,Lung_Opacity/images/Lung_Opacity-4361.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4361.png,Lung_Opacity/masks/Lung_Opacity-4361.png,True,False,299,299,L,147.23,931fd34783851506ccaaa92e239dbb9a
Lung_Opacity-4362.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4362.png,Lung_Opacity/images/Lung_Opacity-4362.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4362.png,Lung_Opacity/masks/Lung_Opacity-4362.png,True,False,299,299,L,155.46,64237facb7cc8e0a93663c5697e0a600
Lung_Opacity-4363.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4363.png,Lung_Opacity/images/Lung_Opacity-4363.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4363.png,Lung_Opacity/masks/Lung_Opacity-4363.png,True,False,299,299,L,91.82,e087b803a667e2e530be5445ef6c1477
Lung_Opacity-4364.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4364.png,Lung_Opacity/images/Lung_Opacity-4364.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4364.png,Lung_Opacity/masks/Lung_Opacity-4364.png,True,False,299,299,L,133.35,83d4b40dc99669846158b8d49c336dd8
Lung_Opacity-4365.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4365.png,Lung_Opacity/images/Lung_Opacity-4365.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4365.png,Lung_Opacity/masks/Lung_Opacity-4365.png,True,False,299,299,L,138.31,07847767b8808f2b0ebfde0c38f8365c
Lung_Opacity-4366.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4366.png,Lung_Opacity/images/Lung_Opacity-4366.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4366.png,Lung_Opacity/masks/Lung_Opacity-4366.png,True,False,299,299,L,159.54,72a1d68ee3bb0e0170989840d4ed0bda
Lung_Opacity-4367.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4367.png,Lung_Opacity/images/Lung_Opacity-4367.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4367.png,Lung_Opacity/masks/Lung_Opacity-4367.png,True,False,299,299,L,133.36,bfc187e47eaf647904f3a3b53bb46d80
Lung_Opacity-4368.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4368.png,Lung_Opacity/images/Lung_Opacity-4368.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4368.png,Lung_Opacity/masks/Lung_Opacity-4368.png,True,False,299,299,L,158.6,3f426f4fa112e7068800c8bbd8ea43df
Lung_Opacity-4369.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4369.png,Lung_Opacity/images/Lung_Opacity-4369.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4369.png,Lung_Opacity/masks/Lung_Opacity-4369.png,True,False,299,299,L,133.8,755b9c8b94531c8323b9e5fd7a8e2902
Lung_Opacity-437.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-437.png,Lung_Opacity/images/Lung_Opacity-437.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-437.png,Lung_Opacity/masks/Lung_Opacity-437.png,True,False,299,299,L,134.83,5bf1ffef33d2c25856f3ff48dd259c2a
Lung_Opacity-4370.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4370.png,Lung_Opacity/images/Lung_Opacity-4370.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4370.png,Lung_Opacity/masks/Lung_Opacity-4370.png,True,False,299,299,L,120.76,e41457abc8f4137a522d096e17559e90
Lung_Opacity-4371.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4371.png,Lung_Opacity/images/Lung_Opacity-4371.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4371.png,Lung_Opacity/masks/Lung_Opacity-4371.png,True,False,299,299,L,112.92,169cba862aa8bdf754e243a63beb8e45
Lung_Opacity-4372.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4372.png,Lung_Opacity/images/Lung_Opacity-4372.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4372.png,Lung_Opacity/masks/Lung_Opacity-4372.png,True,False,299,299,L,115.58,45e9771f24a65a8a4d0ccbe131e0999f
Lung_Opacity-4373.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4373.png,Lung_Opacity/images/Lung_Opacity-4373.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4373.png,Lung_Opacity/masks/Lung_Opacity-4373.png,True,False,299,299,L,103.97,384272d7c4abfe36c374926ba65dde7f
Lung_Opacity-4374.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4374.png,Lung_Opacity/images/Lung_Opacity-4374.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4374.png,Lung_Opacity/masks/Lung_Opacity-4374.png,True,False,299,299,L,122.25,4571cfa02fc0c7750f1fa4f70530d7ae
Lung_Opacity-4375.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4375.png,Lung_Opacity/images/Lung_Opacity-4375.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4375.png,Lung_Opacity/masks/Lung_Opacity-4375.png,True,False,299,299,L,116.59,78f7eb27ed40db15da01d4b5047ca24b
Lung_Opacity-4376.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4376.png,Lung_Opacity/images/Lung_Opacity-4376.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4376.png,Lung_Opacity/masks/Lung_Opacity-4376.png,True,False,299,299,L,165.59,56e6723266a9e6d62a4dfd673114a60c
Lung_Opacity-4377.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4377.png,Lung_Opacity/images/Lung_Opacity-4377.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4377.png,Lung_Opacity/masks/Lung_Opacity-4377.png,True,False,299,299,L,131.84,438d9250c42f2cee2d0812ec06eb4281
Lung_Opacity-4378.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4378.png,Lung_Opacity/images/Lung_Opacity-4378.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4378.png,Lung_Opacity/masks/Lung_Opacity-4378.png,True,False,299,299,L,134.37,5657fbe38d01dc800bc38088faedc8fe
Lung_Opacity-4379.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4379.png,Lung_Opacity/images/Lung_Opacity-4379.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4379.png,Lung_Opacity/masks/Lung_Opacity-4379.png,True,False,299,299,L,122.36,59fac994917052f8f9fe7ffa82f07b6b
Lung_Opacity-438.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-438.png,Lung_Opacity/images/Lung_Opacity-438.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-438.png,Lung_Opacity/masks/Lung_Opacity-438.png,True,False,299,299,L,137.14,9bd1faba56520189459dea0e1e83ecd0
Lung_Opacity-4380.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4380.png,Lung_Opacity/images/Lung_Opacity-4380.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4380.png,Lung_Opacity/masks/Lung_Opacity-4380.png,True,False,299,299,L,120.95,6df7d69fb5140c185bf4cc06ed2062df
Lung_Opacity-4381.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4381.png,Lung_Opacity/images/Lung_Opacity-4381.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4381.png,Lung_Opacity/masks/Lung_Opacity-4381.png,True,False,299,299,L,116.15,d49155292c045dea20134faf1713b602
Lung_Opacity-4382.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4382.png,Lung_Opacity/images/Lung_Opacity-4382.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4382.png,Lung_Opacity/masks/Lung_Opacity-4382.png,True,False,299,299,L,111.99,b3e243e815a06ed739edeea720afa20e
Lung_Opacity-4383.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4383.png,Lung_Opacity/images/Lung_Opacity-4383.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4383.png,Lung_Opacity/masks/Lung_Opacity-4383.png,True,False,299,299,L,156.28,aa2ec3f86385f48dbe8e93eaf8bf4812
Lung_Opacity-4384.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4384.png,Lung_Opacity/images/Lung_Opacity-4384.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4384.png,Lung_Opacity/masks/Lung_Opacity-4384.png,True,False,299,299,L,182.34,e4a95729ca39fb067b66dc9803036341
Lung_Opacity-4385.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4385.png,Lung_Opacity/images/Lung_Opacity-4385.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4385.png,Lung_Opacity/masks/Lung_Opacity-4385.png,True,False,299,299,L,147.93,e91c304595763610ad2fd61f25bbd781
Lung_Opacity-4386.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4386.png,Lung_Opacity/images/Lung_Opacity-4386.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4386.png,Lung_Opacity/masks/Lung_Opacity-4386.png,True,False,299,299,L,117.8,101b9be6d4dd90734c8922e9834d22a5
Lung_Opacity-4387.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4387.png,Lung_Opacity/images/Lung_Opacity-4387.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4387.png,Lung_Opacity/masks/Lung_Opacity-4387.png,True,False,299,299,L,114.34,67e2c4c7cb2f3e4a755c851b23a5811d
Lung_Opacity-4388.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4388.png,Lung_Opacity/images/Lung_Opacity-4388.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4388.png,Lung_Opacity/masks/Lung_Opacity-4388.png,True,False,299,299,L,113.58,be6622f45b197ac700949d17dfab51cb
Lung_Opacity-4389.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4389.png,Lung_Opacity/images/Lung_Opacity-4389.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4389.png,Lung_Opacity/masks/Lung_Opacity-4389.png,True,False,299,299,L,115.96,54ccb9d18f92021bf176eccaba9cbf56
Lung_Opacity-439.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-439.png,Lung_Opacity/images/Lung_Opacity-439.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-439.png,Lung_Opacity/masks/Lung_Opacity-439.png,True,False,299,299,L,106.76,03dd0a139856960a804d563f3e0d906d
Lung_Opacity-4390.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4390.png,Lung_Opacity/images/Lung_Opacity-4390.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4390.png,Lung_Opacity/masks/Lung_Opacity-4390.png,True,False,299,299,L,156.79,bcc9b3bb3bc0f41ca7edd17b80661e12
Lung_Opacity-4391.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4391.png,Lung_Opacity/images/Lung_Opacity-4391.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4391.png,Lung_Opacity/masks/Lung_Opacity-4391.png,True,False,299,299,L,145.19,03500df1f61ce51e526f1364439c3b03
Lung_Opacity-4392.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4392.png,Lung_Opacity/images/Lung_Opacity-4392.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4392.png,Lung_Opacity/masks/Lung_Opacity-4392.png,True,False,299,299,L,110.82,6c7a322dc18a60a78bbdfd374c38358e
Lung_Opacity-4393.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4393.png,Lung_Opacity/images/Lung_Opacity-4393.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4393.png,Lung_Opacity/masks/Lung_Opacity-4393.png,True,False,299,299,L,168.95,6f08ed4cbc7d87fa415732ad6c938d0c
Lung_Opacity-4394.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4394.png,Lung_Opacity/images/Lung_Opacity-4394.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4394.png,Lung_Opacity/masks/Lung_Opacity-4394.png,True,False,299,299,L,129.97,db9aa2879c7f8f1941a3d2daed0c3a99
Lung_Opacity-4395.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4395.png,Lung_Opacity/images/Lung_Opacity-4395.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4395.png,Lung_Opacity/masks/Lung_Opacity-4395.png,True,False,299,299,L,123.8,b91db6129ddb269d9734c6adcf381064
Lung_Opacity-4396.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4396.png,Lung_Opacity/images/Lung_Opacity-4396.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4396.png,Lung_Opacity/masks/Lung_Opacity-4396.png,True,False,299,299,L,174.1,fa09acab03d052af32e5882637b04500
Lung_Opacity-4397.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4397.png,Lung_Opacity/images/Lung_Opacity-4397.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4397.png,Lung_Opacity/masks/Lung_Opacity-4397.png,True,False,299,299,L,111.97,b9a2a02aef4f51904ec6112426ed0f06
Lung_Opacity-4398.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4398.png,Lung_Opacity/images/Lung_Opacity-4398.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4398.png,Lung_Opacity/masks/Lung_Opacity-4398.png,True,False,299,299,L,134.26,7a2273b94558e5f87839cf3baef747a5
Lung_Opacity-4399.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4399.png,Lung_Opacity/images/Lung_Opacity-4399.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4399.png,Lung_Opacity/masks/Lung_Opacity-4399.png,True,False,299,299,L,71.0,d37114e3b1223ca61a8da9d2eb2925aa
Lung_Opacity-44.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-44.png,Lung_Opacity/images/Lung_Opacity-44.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-44.png,Lung_Opacity/masks/Lung_Opacity-44.png,True,False,299,299,L,113.34,8a585c7d9c61bd18e3524984d09282db
Lung_Opacity-440.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-440.png,Lung_Opacity/images/Lung_Opacity-440.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-440.png,Lung_Opacity/masks/Lung_Opacity-440.png,True,False,299,299,L,111.9,424db17ea55d41bc73116dbfb0b5e52c
Lung_Opacity-4400.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4400.png,Lung_Opacity/images/Lung_Opacity-4400.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4400.png,Lung_Opacity/masks/Lung_Opacity-4400.png,True,False,299,299,L,97.45,0d458f5564528cd3137e332e1300ecb5
Lung_Opacity-4401.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4401.png,Lung_Opacity/images/Lung_Opacity-4401.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4401.png,Lung_Opacity/masks/Lung_Opacity-4401.png,True,False,299,299,L,107.33,89b5c4b974b6c63a5614285c55d08a8a
Lung_Opacity-4402.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4402.png,Lung_Opacity/images/Lung_Opacity-4402.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4402.png,Lung_Opacity/masks/Lung_Opacity-4402.png,True,False,299,299,L,117.18,9f4530a32885310ed6ede16311bda23f
Lung_Opacity-4403.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4403.png,Lung_Opacity/images/Lung_Opacity-4403.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4403.png,Lung_Opacity/masks/Lung_Opacity-4403.png,True,False,299,299,L,106.44,0d616a01ec82f58326ab0e750140647d
Lung_Opacity-4404.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4404.png,Lung_Opacity/images/Lung_Opacity-4404.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4404.png,Lung_Opacity/masks/Lung_Opacity-4404.png,True,False,299,299,L,106.48,9c0dd59aa1e93927ffde386daa8bf0f3
Lung_Opacity-4405.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4405.png,Lung_Opacity/images/Lung_Opacity-4405.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4405.png,Lung_Opacity/masks/Lung_Opacity-4405.png,True,False,299,299,L,114.41,e44f0cf0c1fed70838f3377ac4e6b137
Lung_Opacity-4406.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4406.png,Lung_Opacity/images/Lung_Opacity-4406.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4406.png,Lung_Opacity/masks/Lung_Opacity-4406.png,True,False,299,299,L,113.73,81722c10975dd4fe013dc510ae4a6f2c
Lung_Opacity-4407.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4407.png,Lung_Opacity/images/Lung_Opacity-4407.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4407.png,Lung_Opacity/masks/Lung_Opacity-4407.png,True,False,299,299,L,139.48,fd6eadf2db66f728d6651b7a18ca823a
Lung_Opacity-4408.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4408.png,Lung_Opacity/images/Lung_Opacity-4408.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4408.png,Lung_Opacity/masks/Lung_Opacity-4408.png,True,False,299,299,L,108.89,c2a462386939c2db3dfe26a6cfe567f6
Lung_Opacity-4409.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4409.png,Lung_Opacity/images/Lung_Opacity-4409.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4409.png,Lung_Opacity/masks/Lung_Opacity-4409.png,True,False,299,299,L,114.44,1345d26cac19293296f953fa338dbcc1
Lung_Opacity-441.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-441.png,Lung_Opacity/images/Lung_Opacity-441.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-441.png,Lung_Opacity/masks/Lung_Opacity-441.png,True,False,299,299,L,126.98,f55f2ad28db7b02e1b87af0559311aef
Lung_Opacity-4410.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4410.png,Lung_Opacity/images/Lung_Opacity-4410.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4410.png,Lung_Opacity/masks/Lung_Opacity-4410.png,True,False,299,299,L,100.14,bd13e7ed68ec3464100d7c60feda387d
Lung_Opacity-4411.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4411.png,Lung_Opacity/images/Lung_Opacity-4411.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4411.png,Lung_Opacity/masks/Lung_Opacity-4411.png,True,False,299,299,L,135.38,b1b2ec3e819f426ae49db1c4fc662734
Lung_Opacity-4412.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4412.png,Lung_Opacity/images/Lung_Opacity-4412.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4412.png,Lung_Opacity/masks/Lung_Opacity-4412.png,True,False,299,299,L,91.87,f296a2c572366c92b7d280e0a49d440d
Lung_Opacity-4413.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4413.png,Lung_Opacity/images/Lung_Opacity-4413.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4413.png,Lung_Opacity/masks/Lung_Opacity-4413.png,True,False,299,299,L,120.15,cb468105a5f75c526ef223e6487f9e0d
Lung_Opacity-4414.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4414.png,Lung_Opacity/images/Lung_Opacity-4414.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4414.png,Lung_Opacity/masks/Lung_Opacity-4414.png,True,False,299,299,L,129.34,18f34d79c7d307f49fdb4d2211ea3331
Lung_Opacity-4415.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4415.png,Lung_Opacity/images/Lung_Opacity-4415.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4415.png,Lung_Opacity/masks/Lung_Opacity-4415.png,True,False,299,299,L,128.69,6663cc47a5c31af750e182f387137169
Lung_Opacity-4416.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4416.png,Lung_Opacity/images/Lung_Opacity-4416.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4416.png,Lung_Opacity/masks/Lung_Opacity-4416.png,True,False,299,299,L,116.9,9ae08583890475c6aebdd9ac8a92716a
Lung_Opacity-4417.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4417.png,Lung_Opacity/images/Lung_Opacity-4417.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4417.png,Lung_Opacity/masks/Lung_Opacity-4417.png,True,False,299,299,L,157.32,4da2599b348fc09703d72af89af1271e
Lung_Opacity-4418.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4418.png,Lung_Opacity/images/Lung_Opacity-4418.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4418.png,Lung_Opacity/masks/Lung_Opacity-4418.png,True,False,299,299,L,142.34,4f230a05298fac5179c81f96d91e74c1
Lung_Opacity-4419.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4419.png,Lung_Opacity/images/Lung_Opacity-4419.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4419.png,Lung_Opacity/masks/Lung_Opacity-4419.png,True,False,299,299,L,127.5,ed3024d9e05d8f87dcea63385c80383e
Lung_Opacity-442.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-442.png,Lung_Opacity/images/Lung_Opacity-442.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-442.png,Lung_Opacity/masks/Lung_Opacity-442.png,True,False,299,299,L,113.29,8cba70d06d01f69d181f2895f41b2cf7
Lung_Opacity-4420.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4420.png,Lung_Opacity/images/Lung_Opacity-4420.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4420.png,Lung_Opacity/masks/Lung_Opacity-4420.png,True,False,299,299,L,118.37,2d3665884eb934b7e3fd1d594b239fd0
Lung_Opacity-4421.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4421.png,Lung_Opacity/images/Lung_Opacity-4421.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4421.png,Lung_Opacity/masks/Lung_Opacity-4421.png,True,False,299,299,L,150.1,27757fae81b859d05b373351d9381f05
Lung_Opacity-4422.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4422.png,Lung_Opacity/images/Lung_Opacity-4422.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4422.png,Lung_Opacity/masks/Lung_Opacity-4422.png,True,False,299,299,L,159.82,9e5851f9a4c51baea7e8b275fac7dc4f
Lung_Opacity-4423.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4423.png,Lung_Opacity/images/Lung_Opacity-4423.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4423.png,Lung_Opacity/masks/Lung_Opacity-4423.png,True,False,299,299,L,150.26,89520a29c1a670745aa29d2f6611add3
Lung_Opacity-4424.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4424.png,Lung_Opacity/images/Lung_Opacity-4424.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4424.png,Lung_Opacity/masks/Lung_Opacity-4424.png,True,False,299,299,L,118.04,c97d88d016533147f1571fbeb16ef8b1
Lung_Opacity-4425.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4425.png,Lung_Opacity/images/Lung_Opacity-4425.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4425.png,Lung_Opacity/masks/Lung_Opacity-4425.png,True,False,299,299,L,122.91,9aee13e2fc467aa33c98593f77ad6141
Lung_Opacity-4426.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4426.png,Lung_Opacity/images/Lung_Opacity-4426.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4426.png,Lung_Opacity/masks/Lung_Opacity-4426.png,True,False,299,299,L,107.88,207862775c21c4d5e3d188a8774f4e56
Lung_Opacity-4427.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4427.png,Lung_Opacity/images/Lung_Opacity-4427.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4427.png,Lung_Opacity/masks/Lung_Opacity-4427.png,True,False,299,299,L,112.93,6b0129fd84710f261120f83e1f5ca221
Lung_Opacity-4428.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4428.png,Lung_Opacity/images/Lung_Opacity-4428.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4428.png,Lung_Opacity/masks/Lung_Opacity-4428.png,True,False,299,299,L,152.99,f7da7ac024195c3ff2dd8df5ab1afaef
Lung_Opacity-4429.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4429.png,Lung_Opacity/images/Lung_Opacity-4429.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4429.png,Lung_Opacity/masks/Lung_Opacity-4429.png,True,False,299,299,L,109.12,37e8712d574ffc202025dbfa7c0f85c6
Lung_Opacity-443.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-443.png,Lung_Opacity/images/Lung_Opacity-443.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-443.png,Lung_Opacity/masks/Lung_Opacity-443.png,True,False,299,299,L,110.77,c212cfd70c37eae3288ea22e9f77bf15
Lung_Opacity-4430.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4430.png,Lung_Opacity/images/Lung_Opacity-4430.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4430.png,Lung_Opacity/masks/Lung_Opacity-4430.png,True,False,299,299,L,158.27,f2247987d005fcfedf0d66ce819e833c
Lung_Opacity-4431.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4431.png,Lung_Opacity/images/Lung_Opacity-4431.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4431.png,Lung_Opacity/masks/Lung_Opacity-4431.png,True,False,299,299,L,116.5,9a9e0177c53a81a1bf460aa6a7dbe95d
Lung_Opacity-4432.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4432.png,Lung_Opacity/images/Lung_Opacity-4432.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4432.png,Lung_Opacity/masks/Lung_Opacity-4432.png,True,False,299,299,L,131.96,c8e4f3c0647c7ccae2e2c557ce71dd05
Lung_Opacity-4433.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4433.png,Lung_Opacity/images/Lung_Opacity-4433.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4433.png,Lung_Opacity/masks/Lung_Opacity-4433.png,True,False,299,299,L,156.93,883f46d59cce66e8e472c4f1461d5b9d
Lung_Opacity-4434.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4434.png,Lung_Opacity/images/Lung_Opacity-4434.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4434.png,Lung_Opacity/masks/Lung_Opacity-4434.png,True,False,299,299,L,122.12,f79c38aeabba9892476308479173bbd3
Lung_Opacity-4435.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4435.png,Lung_Opacity/images/Lung_Opacity-4435.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4435.png,Lung_Opacity/masks/Lung_Opacity-4435.png,True,False,299,299,L,112.3,67b82de13132fefd3d5f11251754fb2b
Lung_Opacity-4436.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4436.png,Lung_Opacity/images/Lung_Opacity-4436.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4436.png,Lung_Opacity/masks/Lung_Opacity-4436.png,True,False,299,299,L,119.48,fefc54cb5321603e0d801e6e829be04f
Lung_Opacity-4437.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4437.png,Lung_Opacity/images/Lung_Opacity-4437.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4437.png,Lung_Opacity/masks/Lung_Opacity-4437.png,True,False,299,299,L,117.01,3685c4d866fc298f70c3ef22ada61c92
Lung_Opacity-4438.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4438.png,Lung_Opacity/images/Lung_Opacity-4438.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4438.png,Lung_Opacity/masks/Lung_Opacity-4438.png,True,False,299,299,L,137.05,eb31460f6a6912a2703ffe0d728727bc
Lung_Opacity-4439.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4439.png,Lung_Opacity/images/Lung_Opacity-4439.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4439.png,Lung_Opacity/masks/Lung_Opacity-4439.png,True,False,299,299,L,116.87,c5ef3d64c91b8145fefe95ec585cfd21
Lung_Opacity-444.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-444.png,Lung_Opacity/images/Lung_Opacity-444.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-444.png,Lung_Opacity/masks/Lung_Opacity-444.png,True,False,299,299,L,150.37,723442f127e36b22fe059c6d3e10f421
Lung_Opacity-4440.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4440.png,Lung_Opacity/images/Lung_Opacity-4440.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4440.png,Lung_Opacity/masks/Lung_Opacity-4440.png,True,False,299,299,L,113.48,523bad0e30c902916b46a00f979b8baf
Lung_Opacity-4441.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4441.png,Lung_Opacity/images/Lung_Opacity-4441.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4441.png,Lung_Opacity/masks/Lung_Opacity-4441.png,True,False,299,299,L,126.45,73b287816ec24d8d37461474ddee7954
Lung_Opacity-4442.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4442.png,Lung_Opacity/images/Lung_Opacity-4442.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4442.png,Lung_Opacity/masks/Lung_Opacity-4442.png,True,False,299,299,L,116.35,242ad58a5defcaab87c5d5bb91b79a00
Lung_Opacity-4443.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4443.png,Lung_Opacity/images/Lung_Opacity-4443.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4443.png,Lung_Opacity/masks/Lung_Opacity-4443.png,True,False,299,299,L,142.9,60408fb91ec699a9363e090e33f1ff3c
Lung_Opacity-4444.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4444.png,Lung_Opacity/images/Lung_Opacity-4444.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4444.png,Lung_Opacity/masks/Lung_Opacity-4444.png,True,False,299,299,L,92.78,75e4e0dd22fc6a9e8281649e30c07d16
Lung_Opacity-4445.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4445.png,Lung_Opacity/images/Lung_Opacity-4445.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4445.png,Lung_Opacity/masks/Lung_Opacity-4445.png,True,False,299,299,L,151.89,0041a5c2281ce54f3e4840b166733a9c
Lung_Opacity-4446.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4446.png,Lung_Opacity/images/Lung_Opacity-4446.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4446.png,Lung_Opacity/masks/Lung_Opacity-4446.png,True,False,299,299,L,101.46,6053be1b9d1abd4e0cdb53d8280a2bcd
Lung_Opacity-4447.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4447.png,Lung_Opacity/images/Lung_Opacity-4447.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4447.png,Lung_Opacity/masks/Lung_Opacity-4447.png,True,False,299,299,L,110.29,572d186a10dc9bbc5096f3f61678ead5
Lung_Opacity-4448.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4448.png,Lung_Opacity/images/Lung_Opacity-4448.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4448.png,Lung_Opacity/masks/Lung_Opacity-4448.png,True,False,299,299,L,109.64,6d9c3272e4f02bc5e6564803cf8b5f6e
Lung_Opacity-4449.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4449.png,Lung_Opacity/images/Lung_Opacity-4449.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4449.png,Lung_Opacity/masks/Lung_Opacity-4449.png,True,False,299,299,L,152.63,37bfebcc1e947f44f34bd15cb58797d6
Lung_Opacity-445.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-445.png,Lung_Opacity/images/Lung_Opacity-445.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-445.png,Lung_Opacity/masks/Lung_Opacity-445.png,True,False,299,299,L,101.14,68a131ac799366a70465f05c0d2dbb61
Lung_Opacity-4450.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4450.png,Lung_Opacity/images/Lung_Opacity-4450.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4450.png,Lung_Opacity/masks/Lung_Opacity-4450.png,True,False,299,299,L,107.04,f623c253a93f0e742fbbfbf11b4a32a2
Lung_Opacity-4451.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4451.png,Lung_Opacity/images/Lung_Opacity-4451.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4451.png,Lung_Opacity/masks/Lung_Opacity-4451.png,True,False,299,299,L,111.09,832feff704daca701b9c162e419ec94c
Lung_Opacity-4452.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4452.png,Lung_Opacity/images/Lung_Opacity-4452.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4452.png,Lung_Opacity/masks/Lung_Opacity-4452.png,True,False,299,299,L,157.76,abd749fdaa5432ff97ed08abbe185e02
Lung_Opacity-4453.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4453.png,Lung_Opacity/images/Lung_Opacity-4453.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4453.png,Lung_Opacity/masks/Lung_Opacity-4453.png,True,False,299,299,L,112.3,69b3178eb420a5d4c191b86ebc55ebc6
Lung_Opacity-4454.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4454.png,Lung_Opacity/images/Lung_Opacity-4454.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4454.png,Lung_Opacity/masks/Lung_Opacity-4454.png,True,False,299,299,L,99.85,407e4723c7fa64ef975f67f31d0a7c3b
Lung_Opacity-4455.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4455.png,Lung_Opacity/images/Lung_Opacity-4455.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4455.png,Lung_Opacity/masks/Lung_Opacity-4455.png,True,False,299,299,L,107.08,6ee64ee613c9020d934587111967db0b
Lung_Opacity-4456.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4456.png,Lung_Opacity/images/Lung_Opacity-4456.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4456.png,Lung_Opacity/masks/Lung_Opacity-4456.png,True,False,299,299,L,110.8,a8f6f686e753fe2b3ddb3adde4817681
Lung_Opacity-4457.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4457.png,Lung_Opacity/images/Lung_Opacity-4457.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4457.png,Lung_Opacity/masks/Lung_Opacity-4457.png,True,False,299,299,L,156.12,6379a491ec3bdc1aaf376c778f5b7e01
Lung_Opacity-4458.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4458.png,Lung_Opacity/images/Lung_Opacity-4458.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4458.png,Lung_Opacity/masks/Lung_Opacity-4458.png,True,False,299,299,L,110.89,15393b183e34a7736bc55b05d278c318
Lung_Opacity-4459.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4459.png,Lung_Opacity/images/Lung_Opacity-4459.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4459.png,Lung_Opacity/masks/Lung_Opacity-4459.png,True,False,299,299,L,125.78,2e506a15c643edf2d6f7f9e4b3d6d6f4
Lung_Opacity-446.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-446.png,Lung_Opacity/images/Lung_Opacity-446.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-446.png,Lung_Opacity/masks/Lung_Opacity-446.png,True,False,299,299,L,149.51,fea28240aede6dc2323c23484beede61
Lung_Opacity-4460.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4460.png,Lung_Opacity/images/Lung_Opacity-4460.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4460.png,Lung_Opacity/masks/Lung_Opacity-4460.png,True,False,299,299,L,121.38,6c9363e5a6402ba809e25a401bd4e133
Lung_Opacity-4461.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4461.png,Lung_Opacity/images/Lung_Opacity-4461.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4461.png,Lung_Opacity/masks/Lung_Opacity-4461.png,True,False,299,299,L,139.91,13e9e320dd4551e52ac46cfd126c59d4
Lung_Opacity-4462.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4462.png,Lung_Opacity/images/Lung_Opacity-4462.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4462.png,Lung_Opacity/masks/Lung_Opacity-4462.png,True,False,299,299,L,101.15,b89020a8ef135bf70d4b7959b75ae96e
Lung_Opacity-4463.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4463.png,Lung_Opacity/images/Lung_Opacity-4463.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4463.png,Lung_Opacity/masks/Lung_Opacity-4463.png,True,False,299,299,L,107.1,24b014c77813beef900200901657a3ee
Lung_Opacity-4464.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4464.png,Lung_Opacity/images/Lung_Opacity-4464.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4464.png,Lung_Opacity/masks/Lung_Opacity-4464.png,True,False,299,299,L,107.06,b3f55d5d491ac13a7693e13a646b4cf5
Lung_Opacity-4465.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4465.png,Lung_Opacity/images/Lung_Opacity-4465.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4465.png,Lung_Opacity/masks/Lung_Opacity-4465.png,True,False,299,299,L,125.46,071075646c85c5b5f822e7c909966b6a
Lung_Opacity-4466.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4466.png,Lung_Opacity/images/Lung_Opacity-4466.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4466.png,Lung_Opacity/masks/Lung_Opacity-4466.png,True,False,299,299,L,165.09,8df1f9a1942453e312ed2f192aa7084a
Lung_Opacity-4467.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4467.png,Lung_Opacity/images/Lung_Opacity-4467.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4467.png,Lung_Opacity/masks/Lung_Opacity-4467.png,True,False,299,299,L,149.11,7df540165d8bf45a29e520c3f391fd8d
Lung_Opacity-4468.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4468.png,Lung_Opacity/images/Lung_Opacity-4468.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4468.png,Lung_Opacity/masks/Lung_Opacity-4468.png,True,False,299,299,L,164.62,8cc6e31a493a1460cf7bdec961d8088f
Lung_Opacity-4469.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4469.png,Lung_Opacity/images/Lung_Opacity-4469.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4469.png,Lung_Opacity/masks/Lung_Opacity-4469.png,True,False,299,299,L,123.03,653b034924f59cf56cffd09472a73f20
Lung_Opacity-447.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-447.png,Lung_Opacity/images/Lung_Opacity-447.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-447.png,Lung_Opacity/masks/Lung_Opacity-447.png,True,False,299,299,L,105.9,bc4541466f7ac6e0950f32f675cd502b
Lung_Opacity-4470.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4470.png,Lung_Opacity/images/Lung_Opacity-4470.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4470.png,Lung_Opacity/masks/Lung_Opacity-4470.png,True,False,299,299,L,128.68,2f78b5beb9748c6b48d1f14cb02e4f5f
Lung_Opacity-4471.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4471.png,Lung_Opacity/images/Lung_Opacity-4471.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4471.png,Lung_Opacity/masks/Lung_Opacity-4471.png,True,False,299,299,L,82.88,acfa0e6d576e341fd51b456ad77a612d
Lung_Opacity-4472.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4472.png,Lung_Opacity/images/Lung_Opacity-4472.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4472.png,Lung_Opacity/masks/Lung_Opacity-4472.png,True,False,299,299,L,145.7,7f89f7ecd3c68c1e7c21c4dd6320345f
Lung_Opacity-4473.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4473.png,Lung_Opacity/images/Lung_Opacity-4473.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4473.png,Lung_Opacity/masks/Lung_Opacity-4473.png,True,False,299,299,L,152.14,207dd48b71f9b7fecd29d13977a8c15d
Lung_Opacity-4474.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4474.png,Lung_Opacity/images/Lung_Opacity-4474.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4474.png,Lung_Opacity/masks/Lung_Opacity-4474.png,True,False,299,299,L,122.23,9b548fc46f7284302345f5bf9084ba9a
Lung_Opacity-4475.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4475.png,Lung_Opacity/images/Lung_Opacity-4475.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4475.png,Lung_Opacity/masks/Lung_Opacity-4475.png,True,False,299,299,L,134.12,4d5d0ed805c69e796ee1ac74a59bff5e
Lung_Opacity-4476.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4476.png,Lung_Opacity/images/Lung_Opacity-4476.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4476.png,Lung_Opacity/masks/Lung_Opacity-4476.png,True,False,299,299,L,119.79,6ad00c5c3d7fff981b8e982a9306fc4f
Lung_Opacity-4477.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4477.png,Lung_Opacity/images/Lung_Opacity-4477.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4477.png,Lung_Opacity/masks/Lung_Opacity-4477.png,True,False,299,299,L,135.82,2463e79fef1acd8e4c5dd406a0eab02e
Lung_Opacity-4478.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4478.png,Lung_Opacity/images/Lung_Opacity-4478.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4478.png,Lung_Opacity/masks/Lung_Opacity-4478.png,True,False,299,299,L,157.71,768e80c436bead28b71668d353cda3aa
Lung_Opacity-4479.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4479.png,Lung_Opacity/images/Lung_Opacity-4479.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4479.png,Lung_Opacity/masks/Lung_Opacity-4479.png,True,False,299,299,L,111.58,89e64fd35db38e7201c9fb66ae8896c7
Lung_Opacity-448.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-448.png,Lung_Opacity/images/Lung_Opacity-448.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-448.png,Lung_Opacity/masks/Lung_Opacity-448.png,True,False,299,299,L,133.46,aa7dfd454d46059a803982b5a9dec448
Lung_Opacity-4480.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4480.png,Lung_Opacity/images/Lung_Opacity-4480.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4480.png,Lung_Opacity/masks/Lung_Opacity-4480.png,True,False,299,299,L,141.56,65845b1af72d6c850f913184ef2961a9
Lung_Opacity-4481.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4481.png,Lung_Opacity/images/Lung_Opacity-4481.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4481.png,Lung_Opacity/masks/Lung_Opacity-4481.png,True,False,299,299,L,89.65,172a4fd1a2490321bd2a802011096db2
Lung_Opacity-4482.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4482.png,Lung_Opacity/images/Lung_Opacity-4482.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4482.png,Lung_Opacity/masks/Lung_Opacity-4482.png,True,False,299,299,L,107.93,a0b27788212ad4a719150fdeb11950f6
Lung_Opacity-4483.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4483.png,Lung_Opacity/images/Lung_Opacity-4483.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4483.png,Lung_Opacity/masks/Lung_Opacity-4483.png,True,False,299,299,L,142.95,efd2983b3ca9863505d0bbdc794b9580
Lung_Opacity-4484.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4484.png,Lung_Opacity/images/Lung_Opacity-4484.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4484.png,Lung_Opacity/masks/Lung_Opacity-4484.png,True,False,299,299,L,119.05,8619d8da5a43020cac30f2b6e22eb953
Lung_Opacity-4485.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4485.png,Lung_Opacity/images/Lung_Opacity-4485.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4485.png,Lung_Opacity/masks/Lung_Opacity-4485.png,True,False,299,299,L,104.45,3dc351a6ef26be10d59a30db56c599b1
Lung_Opacity-4486.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4486.png,Lung_Opacity/images/Lung_Opacity-4486.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4486.png,Lung_Opacity/masks/Lung_Opacity-4486.png,True,False,299,299,L,165.31,b0ec45844f202757dc7b2fea066b09c8
Lung_Opacity-4487.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4487.png,Lung_Opacity/images/Lung_Opacity-4487.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4487.png,Lung_Opacity/masks/Lung_Opacity-4487.png,True,False,299,299,L,161.24,f41317e6b5db7bcb309caee9344d60e5
Lung_Opacity-4488.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4488.png,Lung_Opacity/images/Lung_Opacity-4488.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4488.png,Lung_Opacity/masks/Lung_Opacity-4488.png,True,False,299,299,L,176.01,77c9faf2f2da74663bf5002f28b4e712
Lung_Opacity-4489.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4489.png,Lung_Opacity/images/Lung_Opacity-4489.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4489.png,Lung_Opacity/masks/Lung_Opacity-4489.png,True,False,299,299,L,157.9,cdc97799719a2bfca74a3d0dcc25c4e2
Lung_Opacity-449.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-449.png,Lung_Opacity/images/Lung_Opacity-449.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-449.png,Lung_Opacity/masks/Lung_Opacity-449.png,True,False,299,299,L,162.46,9663e26ea82df836dc5fffb224cee26d
Lung_Opacity-4490.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4490.png,Lung_Opacity/images/Lung_Opacity-4490.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4490.png,Lung_Opacity/masks/Lung_Opacity-4490.png,True,False,299,299,L,113.55,4cebddf14a198a0407f027c67b0ffacd
Lung_Opacity-4491.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4491.png,Lung_Opacity/images/Lung_Opacity-4491.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4491.png,Lung_Opacity/masks/Lung_Opacity-4491.png,True,False,299,299,L,109.44,ea04eb5a0affca9bb250ea32ea8822b3
Lung_Opacity-4492.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4492.png,Lung_Opacity/images/Lung_Opacity-4492.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4492.png,Lung_Opacity/masks/Lung_Opacity-4492.png,True,False,299,299,L,83.54,8a549ffbb2ee61091d22ffe8fa11b05d
Lung_Opacity-4493.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4493.png,Lung_Opacity/images/Lung_Opacity-4493.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4493.png,Lung_Opacity/masks/Lung_Opacity-4493.png,True,False,299,299,L,155.69,8d699e670334af45493d29d8e2e93ae2
Lung_Opacity-4494.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4494.png,Lung_Opacity/images/Lung_Opacity-4494.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4494.png,Lung_Opacity/masks/Lung_Opacity-4494.png,True,False,299,299,L,160.85,4feaf2b55ed0986762e35144ed540447
Lung_Opacity-4495.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4495.png,Lung_Opacity/images/Lung_Opacity-4495.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4495.png,Lung_Opacity/masks/Lung_Opacity-4495.png,True,False,299,299,L,113.73,792cc60247d8678a9bf6da40cb104e72
Lung_Opacity-4496.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4496.png,Lung_Opacity/images/Lung_Opacity-4496.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4496.png,Lung_Opacity/masks/Lung_Opacity-4496.png,True,False,299,299,L,128.46,2011b041ba9de082d9e16bcdf02021fb
Lung_Opacity-4497.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4497.png,Lung_Opacity/images/Lung_Opacity-4497.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4497.png,Lung_Opacity/masks/Lung_Opacity-4497.png,True,False,299,299,L,118.53,afd4926949878c55aa27e998f79c6121
Lung_Opacity-4498.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4498.png,Lung_Opacity/images/Lung_Opacity-4498.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4498.png,Lung_Opacity/masks/Lung_Opacity-4498.png,True,False,299,299,L,118.31,224dd954ee56a4a205c18281044161fb
Lung_Opacity-4499.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4499.png,Lung_Opacity/images/Lung_Opacity-4499.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4499.png,Lung_Opacity/masks/Lung_Opacity-4499.png,True,False,299,299,L,108.99,3adf238c92b96169edb07f30106b3ad3
Lung_Opacity-45.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-45.png,Lung_Opacity/images/Lung_Opacity-45.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-45.png,Lung_Opacity/masks/Lung_Opacity-45.png,True,False,299,299,L,139.05,3e636e166ae8a40fd69f29d8e28b8789
Lung_Opacity-450.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-450.png,Lung_Opacity/images/Lung_Opacity-450.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-450.png,Lung_Opacity/masks/Lung_Opacity-450.png,True,False,299,299,L,169.72,176302aeeeb13649b70fd5829a16f0ea
Lung_Opacity-4500.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4500.png,Lung_Opacity/images/Lung_Opacity-4500.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4500.png,Lung_Opacity/masks/Lung_Opacity-4500.png,True,False,299,299,L,98.32,c1ee1f228602d0243167f76704f12dd1
Lung_Opacity-4501.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4501.png,Lung_Opacity/images/Lung_Opacity-4501.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4501.png,Lung_Opacity/masks/Lung_Opacity-4501.png,True,False,299,299,L,166.49,13f8fa0267787cab4507c77d7a0f27d7
Lung_Opacity-4502.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4502.png,Lung_Opacity/images/Lung_Opacity-4502.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4502.png,Lung_Opacity/masks/Lung_Opacity-4502.png,True,False,299,299,L,139.1,4216f6161935847048aca9c1f4e3e628
Lung_Opacity-4503.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4503.png,Lung_Opacity/images/Lung_Opacity-4503.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4503.png,Lung_Opacity/masks/Lung_Opacity-4503.png,True,False,299,299,L,125.7,82cfbf5d1a05fb11be12b7705f5895e4
Lung_Opacity-4504.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4504.png,Lung_Opacity/images/Lung_Opacity-4504.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4504.png,Lung_Opacity/masks/Lung_Opacity-4504.png,True,False,299,299,L,108.32,8b9ee8e20063788ea8e956ea11d4fbff
Lung_Opacity-4505.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4505.png,Lung_Opacity/images/Lung_Opacity-4505.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4505.png,Lung_Opacity/masks/Lung_Opacity-4505.png,True,False,299,299,L,117.55,84fe952921608ea5ae6ee83129af75a7
Lung_Opacity-4506.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4506.png,Lung_Opacity/images/Lung_Opacity-4506.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4506.png,Lung_Opacity/masks/Lung_Opacity-4506.png,True,False,299,299,L,106.66,9934927873987d18ed8a8f6856c93c02
Lung_Opacity-4507.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4507.png,Lung_Opacity/images/Lung_Opacity-4507.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4507.png,Lung_Opacity/masks/Lung_Opacity-4507.png,True,False,299,299,L,109.04,95f0981a783840a3b0bfe04c3d277308
Lung_Opacity-4508.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4508.png,Lung_Opacity/images/Lung_Opacity-4508.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4508.png,Lung_Opacity/masks/Lung_Opacity-4508.png,True,False,299,299,L,134.66,493e63319e8139bda7fce25dfc1fb96a
Lung_Opacity-4509.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4509.png,Lung_Opacity/images/Lung_Opacity-4509.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4509.png,Lung_Opacity/masks/Lung_Opacity-4509.png,True,False,299,299,L,117.36,f056f0c73094595a8c1c5f011a321224
Lung_Opacity-451.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-451.png,Lung_Opacity/images/Lung_Opacity-451.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-451.png,Lung_Opacity/masks/Lung_Opacity-451.png,True,False,299,299,L,124.33,16b74b885fb0a8f658ddec1e0f07dd68
Lung_Opacity-4510.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4510.png,Lung_Opacity/images/Lung_Opacity-4510.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4510.png,Lung_Opacity/masks/Lung_Opacity-4510.png,True,False,299,299,L,125.61,b5e9b7a4c656d4d5103f58c1bae7140f
Lung_Opacity-4511.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4511.png,Lung_Opacity/images/Lung_Opacity-4511.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4511.png,Lung_Opacity/masks/Lung_Opacity-4511.png,True,False,299,299,L,126.01,03d9e0706b08d98862830683cf89fa2f
Lung_Opacity-4512.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4512.png,Lung_Opacity/images/Lung_Opacity-4512.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4512.png,Lung_Opacity/masks/Lung_Opacity-4512.png,True,False,299,299,L,154.61,e92755253986d9f514bf315cd9a48b1e
Lung_Opacity-4513.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4513.png,Lung_Opacity/images/Lung_Opacity-4513.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4513.png,Lung_Opacity/masks/Lung_Opacity-4513.png,True,False,299,299,L,101.72,244c3636a4bf93e46a64bc4b366acc46
Lung_Opacity-4514.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4514.png,Lung_Opacity/images/Lung_Opacity-4514.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4514.png,Lung_Opacity/masks/Lung_Opacity-4514.png,True,False,299,299,L,121.44,d1a18fe3a374720c905e44032d10d161
Lung_Opacity-4515.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4515.png,Lung_Opacity/images/Lung_Opacity-4515.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4515.png,Lung_Opacity/masks/Lung_Opacity-4515.png,True,False,299,299,L,115.16,c8e249cac313766075117b46ff48656c
Lung_Opacity-4516.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4516.png,Lung_Opacity/images/Lung_Opacity-4516.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4516.png,Lung_Opacity/masks/Lung_Opacity-4516.png,True,False,299,299,L,118.5,af976183dbd0016fc15ad3a4bf21c70f
Lung_Opacity-4517.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4517.png,Lung_Opacity/images/Lung_Opacity-4517.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4517.png,Lung_Opacity/masks/Lung_Opacity-4517.png,True,False,299,299,L,149.32,8f5765659b67f4db402a0e1283c10e17
Lung_Opacity-4518.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4518.png,Lung_Opacity/images/Lung_Opacity-4518.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4518.png,Lung_Opacity/masks/Lung_Opacity-4518.png,True,False,299,299,L,144.79,7c5baa0f566acec86302370acae002b2
Lung_Opacity-4519.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4519.png,Lung_Opacity/images/Lung_Opacity-4519.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4519.png,Lung_Opacity/masks/Lung_Opacity-4519.png,True,False,299,299,L,130.46,ac759026813d18fb1c79dc85c76846d4
Lung_Opacity-452.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-452.png,Lung_Opacity/images/Lung_Opacity-452.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-452.png,Lung_Opacity/masks/Lung_Opacity-452.png,True,False,299,299,L,150.48,b383a585cb668a2e9ba20067873f7f69
Lung_Opacity-4520.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4520.png,Lung_Opacity/images/Lung_Opacity-4520.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4520.png,Lung_Opacity/masks/Lung_Opacity-4520.png,True,False,299,299,L,114.92,080e199fc7673b596dcebc93b503eb0b
Lung_Opacity-4521.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4521.png,Lung_Opacity/images/Lung_Opacity-4521.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4521.png,Lung_Opacity/masks/Lung_Opacity-4521.png,True,False,299,299,L,152.13,059700d4ce62b2f082e71cdb45dd703f
Lung_Opacity-4522.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4522.png,Lung_Opacity/images/Lung_Opacity-4522.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4522.png,Lung_Opacity/masks/Lung_Opacity-4522.png,True,False,299,299,L,132.69,882d564a181efc1095af9c4cac001f9f
Lung_Opacity-4523.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4523.png,Lung_Opacity/images/Lung_Opacity-4523.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4523.png,Lung_Opacity/masks/Lung_Opacity-4523.png,True,False,299,299,L,120.19,38ffad0184d632f32fc8080e97fedb9d
Lung_Opacity-4524.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4524.png,Lung_Opacity/images/Lung_Opacity-4524.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4524.png,Lung_Opacity/masks/Lung_Opacity-4524.png,True,False,299,299,L,126.34,395208a0f21f94d4e08b8370f49ea7de
Lung_Opacity-4525.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4525.png,Lung_Opacity/images/Lung_Opacity-4525.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4525.png,Lung_Opacity/masks/Lung_Opacity-4525.png,True,False,299,299,L,155.65,beb32268c4f39e4f0a6ddd8ebb6ac875
Lung_Opacity-4526.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4526.png,Lung_Opacity/images/Lung_Opacity-4526.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4526.png,Lung_Opacity/masks/Lung_Opacity-4526.png,True,False,299,299,L,124.29,de77c6b970eb9b8ea377fb766fab16b7
Lung_Opacity-4527.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4527.png,Lung_Opacity/images/Lung_Opacity-4527.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4527.png,Lung_Opacity/masks/Lung_Opacity-4527.png,True,False,299,299,L,109.0,812f792f0de5441ac30fdf0a11f5bc06
Lung_Opacity-4528.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4528.png,Lung_Opacity/images/Lung_Opacity-4528.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4528.png,Lung_Opacity/masks/Lung_Opacity-4528.png,True,False,299,299,L,154.03,20501f33859f1f67eb02dd11a206e778
Lung_Opacity-4529.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4529.png,Lung_Opacity/images/Lung_Opacity-4529.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4529.png,Lung_Opacity/masks/Lung_Opacity-4529.png,True,False,299,299,L,139.89,8c5f40724bdcae2b911ee335e7cad105
Lung_Opacity-453.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-453.png,Lung_Opacity/images/Lung_Opacity-453.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-453.png,Lung_Opacity/masks/Lung_Opacity-453.png,True,False,299,299,L,118.73,2ed6d31ee52a3eb9a56383f8dd5aa853
Lung_Opacity-4530.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4530.png,Lung_Opacity/images/Lung_Opacity-4530.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4530.png,Lung_Opacity/masks/Lung_Opacity-4530.png,True,False,299,299,L,95.12,404dbc75754c68b151a9fb60b79aff01
Lung_Opacity-4531.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4531.png,Lung_Opacity/images/Lung_Opacity-4531.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4531.png,Lung_Opacity/masks/Lung_Opacity-4531.png,True,False,299,299,L,107.7,93bbc6f9c69e1e530f64a8d1602d5dba
Lung_Opacity-4532.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4532.png,Lung_Opacity/images/Lung_Opacity-4532.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4532.png,Lung_Opacity/masks/Lung_Opacity-4532.png,True,False,299,299,L,106.58,3e4d3fa7d7f65445ae8278d13899bc0f
Lung_Opacity-4533.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4533.png,Lung_Opacity/images/Lung_Opacity-4533.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4533.png,Lung_Opacity/masks/Lung_Opacity-4533.png,True,False,299,299,L,109.48,9e6100794ca2d3fd5a53222a8593d991
Lung_Opacity-4534.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4534.png,Lung_Opacity/images/Lung_Opacity-4534.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4534.png,Lung_Opacity/masks/Lung_Opacity-4534.png,True,False,299,299,L,153.92,a186e0e993a2468ef4cd24c47f17544c
Lung_Opacity-4535.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4535.png,Lung_Opacity/images/Lung_Opacity-4535.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4535.png,Lung_Opacity/masks/Lung_Opacity-4535.png,True,False,299,299,L,137.11,04106156f44342a95439699567b2ecd7
Lung_Opacity-4536.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4536.png,Lung_Opacity/images/Lung_Opacity-4536.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4536.png,Lung_Opacity/masks/Lung_Opacity-4536.png,True,False,299,299,L,136.53,3041b8771b8ba10fa6260978503dbf16
Lung_Opacity-4537.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4537.png,Lung_Opacity/images/Lung_Opacity-4537.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4537.png,Lung_Opacity/masks/Lung_Opacity-4537.png,True,False,299,299,L,83.64,1a0ee50c7fb10a4299ee05a55e139983
Lung_Opacity-4538.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4538.png,Lung_Opacity/images/Lung_Opacity-4538.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4538.png,Lung_Opacity/masks/Lung_Opacity-4538.png,True,False,299,299,L,121.08,1036940775466a68d5e22d7ac5d1e101
Lung_Opacity-4539.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4539.png,Lung_Opacity/images/Lung_Opacity-4539.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4539.png,Lung_Opacity/masks/Lung_Opacity-4539.png,True,False,299,299,L,115.65,bcddbb1fd741a7c6c2d600a5d8a6587d
Lung_Opacity-454.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-454.png,Lung_Opacity/images/Lung_Opacity-454.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-454.png,Lung_Opacity/masks/Lung_Opacity-454.png,True,False,299,299,L,121.29,0eaf6d82601543ffc1a123597bdba2df
Lung_Opacity-4540.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4540.png,Lung_Opacity/images/Lung_Opacity-4540.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4540.png,Lung_Opacity/masks/Lung_Opacity-4540.png,True,False,299,299,L,169.27,de7abb02fd3308ece5c7c58ff6a1d5c3
Lung_Opacity-4541.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4541.png,Lung_Opacity/images/Lung_Opacity-4541.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4541.png,Lung_Opacity/masks/Lung_Opacity-4541.png,True,False,299,299,L,115.18,6233dd14f3dfa8c3a68bf2179a46fad9
Lung_Opacity-4542.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4542.png,Lung_Opacity/images/Lung_Opacity-4542.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4542.png,Lung_Opacity/masks/Lung_Opacity-4542.png,True,False,299,299,L,115.6,0835f93c768fb20385a66c8a5a144403
Lung_Opacity-4543.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4543.png,Lung_Opacity/images/Lung_Opacity-4543.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4543.png,Lung_Opacity/masks/Lung_Opacity-4543.png,True,False,299,299,L,107.25,b5eaee9f9e9110777b114ecbfe065563
Lung_Opacity-4544.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4544.png,Lung_Opacity/images/Lung_Opacity-4544.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4544.png,Lung_Opacity/masks/Lung_Opacity-4544.png,True,False,299,299,L,113.59,3c1aef7ab75720fd0482e0d4bf474f81
Lung_Opacity-4545.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4545.png,Lung_Opacity/images/Lung_Opacity-4545.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4545.png,Lung_Opacity/masks/Lung_Opacity-4545.png,True,False,299,299,L,121.44,ffcec07278b6ce98d05590e4157137bd
Lung_Opacity-4546.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4546.png,Lung_Opacity/images/Lung_Opacity-4546.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4546.png,Lung_Opacity/masks/Lung_Opacity-4546.png,True,False,299,299,L,155.83,b876196e3c704734d6a5254d4a139605
Lung_Opacity-4547.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4547.png,Lung_Opacity/images/Lung_Opacity-4547.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4547.png,Lung_Opacity/masks/Lung_Opacity-4547.png,True,False,299,299,L,166.52,a5965efba77743980cbadbd013de17e5
Lung_Opacity-4548.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4548.png,Lung_Opacity/images/Lung_Opacity-4548.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4548.png,Lung_Opacity/masks/Lung_Opacity-4548.png,True,False,299,299,L,143.84,e9acbe8bebd96b3e948eda9d674a2f07
Lung_Opacity-4549.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4549.png,Lung_Opacity/images/Lung_Opacity-4549.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4549.png,Lung_Opacity/masks/Lung_Opacity-4549.png,True,False,299,299,L,113.17,a95221a9f0518adcc47720294b693b50
Lung_Opacity-455.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-455.png,Lung_Opacity/images/Lung_Opacity-455.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-455.png,Lung_Opacity/masks/Lung_Opacity-455.png,True,False,299,299,L,118.57,68eca9e49417a42e7759e841b9459bf5
Lung_Opacity-4550.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4550.png,Lung_Opacity/images/Lung_Opacity-4550.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4550.png,Lung_Opacity/masks/Lung_Opacity-4550.png,True,False,299,299,L,115.82,0e2e28fc16309b6a1079c5da111a6327
Lung_Opacity-4551.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4551.png,Lung_Opacity/images/Lung_Opacity-4551.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4551.png,Lung_Opacity/masks/Lung_Opacity-4551.png,True,False,299,299,L,118.1,9df328ec59f4dd70ee3ee62e77029d30
Lung_Opacity-4552.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4552.png,Lung_Opacity/images/Lung_Opacity-4552.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4552.png,Lung_Opacity/masks/Lung_Opacity-4552.png,True,False,299,299,L,134.7,7145f46a4ee062c52021b6120ff9de8e
Lung_Opacity-4553.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4553.png,Lung_Opacity/images/Lung_Opacity-4553.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4553.png,Lung_Opacity/masks/Lung_Opacity-4553.png,True,False,299,299,L,121.63,396253d8776b1f9e42304c418130488f
Lung_Opacity-4554.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4554.png,Lung_Opacity/images/Lung_Opacity-4554.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4554.png,Lung_Opacity/masks/Lung_Opacity-4554.png,True,False,299,299,L,110.88,c8f7dcae62476cb05b6c08c841c97802
Lung_Opacity-4555.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4555.png,Lung_Opacity/images/Lung_Opacity-4555.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4555.png,Lung_Opacity/masks/Lung_Opacity-4555.png,True,False,299,299,L,168.32,a4e0f3a8973feb984e9bd4667157edd4
Lung_Opacity-4556.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4556.png,Lung_Opacity/images/Lung_Opacity-4556.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4556.png,Lung_Opacity/masks/Lung_Opacity-4556.png,True,False,299,299,L,129.3,e65c33c8dd57cadd86f4a26184552f37
Lung_Opacity-4557.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4557.png,Lung_Opacity/images/Lung_Opacity-4557.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4557.png,Lung_Opacity/masks/Lung_Opacity-4557.png,True,False,299,299,L,114.22,5f1dafef93f35e75f4ea7cce03f0cf0b
Lung_Opacity-4558.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4558.png,Lung_Opacity/images/Lung_Opacity-4558.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4558.png,Lung_Opacity/masks/Lung_Opacity-4558.png,True,False,299,299,L,138.97,d325d3e6439447bc8f36a38becbc7461
Lung_Opacity-4559.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4559.png,Lung_Opacity/images/Lung_Opacity-4559.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4559.png,Lung_Opacity/masks/Lung_Opacity-4559.png,True,False,299,299,L,94.47,b18a34a7ce70eedbfe2bb6601c38e183
Lung_Opacity-456.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-456.png,Lung_Opacity/images/Lung_Opacity-456.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-456.png,Lung_Opacity/masks/Lung_Opacity-456.png,True,False,299,299,L,123.82,a17bce783491e1d37f1aae49e8264f31
Lung_Opacity-4560.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4560.png,Lung_Opacity/images/Lung_Opacity-4560.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4560.png,Lung_Opacity/masks/Lung_Opacity-4560.png,True,False,299,299,L,124.29,dcedb9cea04db5b0fc003734cb66614c
Lung_Opacity-4561.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4561.png,Lung_Opacity/images/Lung_Opacity-4561.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4561.png,Lung_Opacity/masks/Lung_Opacity-4561.png,True,False,299,299,L,175.95,065807b037349d293de600160be8ab46
Lung_Opacity-4562.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4562.png,Lung_Opacity/images/Lung_Opacity-4562.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4562.png,Lung_Opacity/masks/Lung_Opacity-4562.png,True,False,299,299,L,148.84,540865497fe15e017c2b49f37817da72
Lung_Opacity-4563.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4563.png,Lung_Opacity/images/Lung_Opacity-4563.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4563.png,Lung_Opacity/masks/Lung_Opacity-4563.png,True,False,299,299,L,126.54,5a33bb5e02cb0e2e92877ea7e55c1e4b
Lung_Opacity-4564.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4564.png,Lung_Opacity/images/Lung_Opacity-4564.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4564.png,Lung_Opacity/masks/Lung_Opacity-4564.png,True,False,299,299,L,117.27,1a4c6364fdbcdee48e84d743b18f01b6
Lung_Opacity-4565.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4565.png,Lung_Opacity/images/Lung_Opacity-4565.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4565.png,Lung_Opacity/masks/Lung_Opacity-4565.png,True,False,299,299,L,108.64,45270cb6e421d8bee80f21b8d8b326b7
Lung_Opacity-4566.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4566.png,Lung_Opacity/images/Lung_Opacity-4566.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4566.png,Lung_Opacity/masks/Lung_Opacity-4566.png,True,False,299,299,L,115.74,623c5faf0d61fd9577f67ec1e7b84ed8
Lung_Opacity-4567.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4567.png,Lung_Opacity/images/Lung_Opacity-4567.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4567.png,Lung_Opacity/masks/Lung_Opacity-4567.png,True,False,299,299,L,126.44,6c1e4946a0d016ed554478d01889ae23
Lung_Opacity-4568.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4568.png,Lung_Opacity/images/Lung_Opacity-4568.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4568.png,Lung_Opacity/masks/Lung_Opacity-4568.png,True,False,299,299,L,125.85,c44311c2967a48ce9f37d41e5226fa8c
Lung_Opacity-4569.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4569.png,Lung_Opacity/images/Lung_Opacity-4569.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4569.png,Lung_Opacity/masks/Lung_Opacity-4569.png,True,False,299,299,L,117.44,628bf8a0ee6f130760147f9ae76688de
Lung_Opacity-457.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-457.png,Lung_Opacity/images/Lung_Opacity-457.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-457.png,Lung_Opacity/masks/Lung_Opacity-457.png,True,False,299,299,L,143.38,03785373fcb79f82180cb452b519d18d
Lung_Opacity-4570.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4570.png,Lung_Opacity/images/Lung_Opacity-4570.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4570.png,Lung_Opacity/masks/Lung_Opacity-4570.png,True,False,299,299,L,129.37,783ce7d0361510c2cddef0c8557db460
Lung_Opacity-4571.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4571.png,Lung_Opacity/images/Lung_Opacity-4571.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4571.png,Lung_Opacity/masks/Lung_Opacity-4571.png,True,False,299,299,L,156.68,81f1b15ed64799ac517293c1d5c1cad6
Lung_Opacity-4572.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4572.png,Lung_Opacity/images/Lung_Opacity-4572.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4572.png,Lung_Opacity/masks/Lung_Opacity-4572.png,True,False,299,299,L,128.24,cccc7cecf9eaf7f6c899167485a323f4
Lung_Opacity-4573.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4573.png,Lung_Opacity/images/Lung_Opacity-4573.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4573.png,Lung_Opacity/masks/Lung_Opacity-4573.png,True,False,299,299,L,137.19,0c8b47b225f24dec04e2a8700423a5e1
Lung_Opacity-4574.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4574.png,Lung_Opacity/images/Lung_Opacity-4574.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4574.png,Lung_Opacity/masks/Lung_Opacity-4574.png,True,False,299,299,L,152.47,ad95af82f4bd40417aa2dc2563abc9f4
Lung_Opacity-4575.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4575.png,Lung_Opacity/images/Lung_Opacity-4575.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4575.png,Lung_Opacity/masks/Lung_Opacity-4575.png,True,False,299,299,L,151.57,9145ab410db9c8c9ac710732b7e67250
Lung_Opacity-4576.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4576.png,Lung_Opacity/images/Lung_Opacity-4576.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4576.png,Lung_Opacity/masks/Lung_Opacity-4576.png,True,False,299,299,L,127.98,240d17eae123a5221559920333c22651
Lung_Opacity-4577.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4577.png,Lung_Opacity/images/Lung_Opacity-4577.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4577.png,Lung_Opacity/masks/Lung_Opacity-4577.png,True,False,299,299,L,121.67,05792103df1107b5a773d7fd34df89d4
Lung_Opacity-4578.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4578.png,Lung_Opacity/images/Lung_Opacity-4578.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4578.png,Lung_Opacity/masks/Lung_Opacity-4578.png,True,False,299,299,L,132.3,bd707608f78c2e5120da358681024839
Lung_Opacity-4579.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4579.png,Lung_Opacity/images/Lung_Opacity-4579.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4579.png,Lung_Opacity/masks/Lung_Opacity-4579.png,True,False,299,299,L,155.93,81149b379cd124570bda6b31f0efb528
Lung_Opacity-458.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-458.png,Lung_Opacity/images/Lung_Opacity-458.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-458.png,Lung_Opacity/masks/Lung_Opacity-458.png,True,False,299,299,L,105.1,35e973bcde6fc5a1be44ff4f605b5d99
Lung_Opacity-4580.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4580.png,Lung_Opacity/images/Lung_Opacity-4580.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4580.png,Lung_Opacity/masks/Lung_Opacity-4580.png,True,False,299,299,L,114.96,4a3af56dafde10cd0a38ad16fa7944e9
Lung_Opacity-4581.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4581.png,Lung_Opacity/images/Lung_Opacity-4581.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4581.png,Lung_Opacity/masks/Lung_Opacity-4581.png,True,False,299,299,L,155.61,62e52274b52a688b9d5441e324939e5f
Lung_Opacity-4582.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4582.png,Lung_Opacity/images/Lung_Opacity-4582.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4582.png,Lung_Opacity/masks/Lung_Opacity-4582.png,True,False,299,299,L,139.1,a8d27fbe403df468ca8e5d6afed6cbf4
Lung_Opacity-4583.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4583.png,Lung_Opacity/images/Lung_Opacity-4583.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4583.png,Lung_Opacity/masks/Lung_Opacity-4583.png,True,False,299,299,L,108.63,8ebb8651aa377f90b3e2efccda7208e6
Lung_Opacity-4584.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4584.png,Lung_Opacity/images/Lung_Opacity-4584.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4584.png,Lung_Opacity/masks/Lung_Opacity-4584.png,True,False,299,299,L,135.57,aae1b01ba4dd3897d1072516b3b43b4f
Lung_Opacity-4585.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4585.png,Lung_Opacity/images/Lung_Opacity-4585.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4585.png,Lung_Opacity/masks/Lung_Opacity-4585.png,True,False,299,299,L,111.4,c51e4c3e0621ecf899ca91d20500cdd7
Lung_Opacity-4586.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4586.png,Lung_Opacity/images/Lung_Opacity-4586.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4586.png,Lung_Opacity/masks/Lung_Opacity-4586.png,True,False,299,299,L,149.91,31e9eb91e27187e74d638a90d9f3d52c
Lung_Opacity-4587.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4587.png,Lung_Opacity/images/Lung_Opacity-4587.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4587.png,Lung_Opacity/masks/Lung_Opacity-4587.png,True,False,299,299,L,134.25,3d841ed950abf1b347e28b1fc63e164d
Lung_Opacity-4588.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4588.png,Lung_Opacity/images/Lung_Opacity-4588.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4588.png,Lung_Opacity/masks/Lung_Opacity-4588.png,True,False,299,299,L,105.21,c8a4a5111b2a5c6d2dd54334059690f6
Lung_Opacity-4589.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4589.png,Lung_Opacity/images/Lung_Opacity-4589.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4589.png,Lung_Opacity/masks/Lung_Opacity-4589.png,True,False,299,299,L,126.86,2ce3baa5d762971b4728fef844043d66
Lung_Opacity-459.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-459.png,Lung_Opacity/images/Lung_Opacity-459.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-459.png,Lung_Opacity/masks/Lung_Opacity-459.png,True,False,299,299,L,117.36,61138d53866ab63d3c6e909c679a65f4
Lung_Opacity-4590.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4590.png,Lung_Opacity/images/Lung_Opacity-4590.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4590.png,Lung_Opacity/masks/Lung_Opacity-4590.png,True,False,299,299,L,111.54,c2585bca424b6a173520922bdd60eb74
Lung_Opacity-4591.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4591.png,Lung_Opacity/images/Lung_Opacity-4591.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4591.png,Lung_Opacity/masks/Lung_Opacity-4591.png,True,False,299,299,L,105.38,ed4ec51f8d1dc3d414a629090310b144
Lung_Opacity-4592.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4592.png,Lung_Opacity/images/Lung_Opacity-4592.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4592.png,Lung_Opacity/masks/Lung_Opacity-4592.png,True,False,299,299,L,101.69,78fa079a7327800932acdf604358d632
Lung_Opacity-4593.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4593.png,Lung_Opacity/images/Lung_Opacity-4593.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4593.png,Lung_Opacity/masks/Lung_Opacity-4593.png,True,False,299,299,L,109.17,76d66b745e62bd9dd401b8a91e8d5f21
Lung_Opacity-4594.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4594.png,Lung_Opacity/images/Lung_Opacity-4594.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4594.png,Lung_Opacity/masks/Lung_Opacity-4594.png,True,False,299,299,L,145.27,e47c2059c97cc8bce6a58d7031d920d4
Lung_Opacity-4595.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4595.png,Lung_Opacity/images/Lung_Opacity-4595.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4595.png,Lung_Opacity/masks/Lung_Opacity-4595.png,True,False,299,299,L,132.22,892a6f2412577de28b3b1a8e3406563d
Lung_Opacity-4596.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4596.png,Lung_Opacity/images/Lung_Opacity-4596.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4596.png,Lung_Opacity/masks/Lung_Opacity-4596.png,True,False,299,299,L,109.39,053261240d422048e591b9195eba4512
Lung_Opacity-4597.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4597.png,Lung_Opacity/images/Lung_Opacity-4597.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4597.png,Lung_Opacity/masks/Lung_Opacity-4597.png,True,False,299,299,L,152.68,cbb0dfb49526bb375ff7575267ff4f4b
Lung_Opacity-4598.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4598.png,Lung_Opacity/images/Lung_Opacity-4598.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4598.png,Lung_Opacity/masks/Lung_Opacity-4598.png,True,False,299,299,L,140.18,d4ef58a56956cf61f1edc1b9385efa2c
Lung_Opacity-4599.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4599.png,Lung_Opacity/images/Lung_Opacity-4599.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4599.png,Lung_Opacity/masks/Lung_Opacity-4599.png,True,False,299,299,L,111.19,8e7e1694e8e35c13d89f797d8fa8316c
Lung_Opacity-46.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-46.png,Lung_Opacity/images/Lung_Opacity-46.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-46.png,Lung_Opacity/masks/Lung_Opacity-46.png,True,False,299,299,L,146.94,276794aa91e71fb8a0ee5da8fa96fc2a
Lung_Opacity-460.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-460.png,Lung_Opacity/images/Lung_Opacity-460.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-460.png,Lung_Opacity/masks/Lung_Opacity-460.png,True,False,299,299,L,152.38,23d82cc4adc039e387545503710dd7f9
Lung_Opacity-4600.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4600.png,Lung_Opacity/images/Lung_Opacity-4600.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4600.png,Lung_Opacity/masks/Lung_Opacity-4600.png,True,False,299,299,L,116.59,688a97973d15d9f81830d684716b4520
Lung_Opacity-4601.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4601.png,Lung_Opacity/images/Lung_Opacity-4601.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4601.png,Lung_Opacity/masks/Lung_Opacity-4601.png,True,False,299,299,L,157.2,4f1b5a366ae84cae2b7940fe3fa7cf1b
Lung_Opacity-4602.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4602.png,Lung_Opacity/images/Lung_Opacity-4602.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4602.png,Lung_Opacity/masks/Lung_Opacity-4602.png,True,False,299,299,L,126.5,0d96ac005fd150dfc7552c0d85cf7779
Lung_Opacity-4603.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4603.png,Lung_Opacity/images/Lung_Opacity-4603.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4603.png,Lung_Opacity/masks/Lung_Opacity-4603.png,True,False,299,299,L,118.0,efbb2eeecfc1c5e30055a91e038c4569
Lung_Opacity-4604.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4604.png,Lung_Opacity/images/Lung_Opacity-4604.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4604.png,Lung_Opacity/masks/Lung_Opacity-4604.png,True,False,299,299,L,137.88,a3a8926d0a8b02d0383a64b8158d91c9
Lung_Opacity-4605.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4605.png,Lung_Opacity/images/Lung_Opacity-4605.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4605.png,Lung_Opacity/masks/Lung_Opacity-4605.png,True,False,299,299,L,110.99,4cdd29b5243652f80b29c6205fbf1fed
Lung_Opacity-4606.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4606.png,Lung_Opacity/images/Lung_Opacity-4606.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4606.png,Lung_Opacity/masks/Lung_Opacity-4606.png,True,False,299,299,L,108.38,fb03addcd88af3ef053afb51410b30ae
Lung_Opacity-4607.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4607.png,Lung_Opacity/images/Lung_Opacity-4607.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4607.png,Lung_Opacity/masks/Lung_Opacity-4607.png,True,False,299,299,L,99.93,5f8e450ae7cd61e48be17a7af9ab8758
Lung_Opacity-4608.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4608.png,Lung_Opacity/images/Lung_Opacity-4608.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4608.png,Lung_Opacity/masks/Lung_Opacity-4608.png,True,False,299,299,L,124.68,566778aab0c2a813a13ca04c94c0318d
Lung_Opacity-4609.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4609.png,Lung_Opacity/images/Lung_Opacity-4609.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4609.png,Lung_Opacity/masks/Lung_Opacity-4609.png,True,False,299,299,L,121.57,18887d18f93e49e6130ba50dd3aae933
Lung_Opacity-461.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-461.png,Lung_Opacity/images/Lung_Opacity-461.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-461.png,Lung_Opacity/masks/Lung_Opacity-461.png,True,False,299,299,L,118.43,8b37fcd86667a9066d5b02d0c6dbc49f
Lung_Opacity-4610.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4610.png,Lung_Opacity/images/Lung_Opacity-4610.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4610.png,Lung_Opacity/masks/Lung_Opacity-4610.png,True,False,299,299,L,119.09,52690a3c1bc17635fd867f6c08fbaace
Lung_Opacity-4611.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4611.png,Lung_Opacity/images/Lung_Opacity-4611.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4611.png,Lung_Opacity/masks/Lung_Opacity-4611.png,True,False,299,299,L,126.47,47e77170ee0f0aef03b7be7257012651
Lung_Opacity-4612.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4612.png,Lung_Opacity/images/Lung_Opacity-4612.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4612.png,Lung_Opacity/masks/Lung_Opacity-4612.png,True,False,299,299,L,124.32,eaf711f69706a92a860a572088368cee
Lung_Opacity-4613.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4613.png,Lung_Opacity/images/Lung_Opacity-4613.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4613.png,Lung_Opacity/masks/Lung_Opacity-4613.png,True,False,299,299,L,173.91,e8ed92c4c5beaf12d4d8ec126a87115d
Lung_Opacity-4614.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4614.png,Lung_Opacity/images/Lung_Opacity-4614.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4614.png,Lung_Opacity/masks/Lung_Opacity-4614.png,True,False,299,299,L,102.45,2252da9c4639d35a09331ccec4b079b3
Lung_Opacity-4615.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4615.png,Lung_Opacity/images/Lung_Opacity-4615.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4615.png,Lung_Opacity/masks/Lung_Opacity-4615.png,True,False,299,299,L,88.17,9ba226c064ac577763fc8fdfe7629b7e
Lung_Opacity-4616.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4616.png,Lung_Opacity/images/Lung_Opacity-4616.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4616.png,Lung_Opacity/masks/Lung_Opacity-4616.png,True,False,299,299,L,161.0,38ed8ad0d83acb138f6b28f6bb75698c
Lung_Opacity-4617.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4617.png,Lung_Opacity/images/Lung_Opacity-4617.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4617.png,Lung_Opacity/masks/Lung_Opacity-4617.png,True,False,299,299,L,113.52,e3107790aa5531588bcb7f62ea80ece6
Lung_Opacity-4618.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4618.png,Lung_Opacity/images/Lung_Opacity-4618.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4618.png,Lung_Opacity/masks/Lung_Opacity-4618.png,True,False,299,299,L,105.87,7cdbc170a55da6679b6fab1960fb7031
Lung_Opacity-4619.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4619.png,Lung_Opacity/images/Lung_Opacity-4619.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4619.png,Lung_Opacity/masks/Lung_Opacity-4619.png,True,False,299,299,L,164.16,82bb6dd04c2a1277db30bb587cb4ab48
Lung_Opacity-462.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-462.png,Lung_Opacity/images/Lung_Opacity-462.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-462.png,Lung_Opacity/masks/Lung_Opacity-462.png,True,False,299,299,L,113.39,7ec423dd229951455612676363d99b74
Lung_Opacity-4620.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4620.png,Lung_Opacity/images/Lung_Opacity-4620.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4620.png,Lung_Opacity/masks/Lung_Opacity-4620.png,True,False,299,299,L,114.39,e6afe7f41dd598bac85685894c145ef4
Lung_Opacity-4621.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4621.png,Lung_Opacity/images/Lung_Opacity-4621.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4621.png,Lung_Opacity/masks/Lung_Opacity-4621.png,True,False,299,299,L,157.36,0d5a2f20c97989266444dce295181719
Lung_Opacity-4622.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4622.png,Lung_Opacity/images/Lung_Opacity-4622.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4622.png,Lung_Opacity/masks/Lung_Opacity-4622.png,True,False,299,299,L,140.43,0afe16abe5fc89a7a52d17df8fe60b72
Lung_Opacity-4623.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4623.png,Lung_Opacity/images/Lung_Opacity-4623.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4623.png,Lung_Opacity/masks/Lung_Opacity-4623.png,True,False,299,299,L,153.76,6db364cb49958c6bedb9a7041ced612b
Lung_Opacity-4624.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4624.png,Lung_Opacity/images/Lung_Opacity-4624.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4624.png,Lung_Opacity/masks/Lung_Opacity-4624.png,True,False,299,299,L,174.86,dc9629591d1baaadc948960bf1419509
Lung_Opacity-4625.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4625.png,Lung_Opacity/images/Lung_Opacity-4625.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4625.png,Lung_Opacity/masks/Lung_Opacity-4625.png,True,False,299,299,L,138.65,20f1f4cbfdbee3c4cb6c8116287d2d0e
Lung_Opacity-4626.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4626.png,Lung_Opacity/images/Lung_Opacity-4626.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4626.png,Lung_Opacity/masks/Lung_Opacity-4626.png,True,False,299,299,L,111.14,628f3848ea9743bef8657a1dc913611e
Lung_Opacity-4627.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4627.png,Lung_Opacity/images/Lung_Opacity-4627.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4627.png,Lung_Opacity/masks/Lung_Opacity-4627.png,True,False,299,299,L,114.21,81332045e6cc03775ac28a97cb909ddf
Lung_Opacity-4628.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4628.png,Lung_Opacity/images/Lung_Opacity-4628.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4628.png,Lung_Opacity/masks/Lung_Opacity-4628.png,True,False,299,299,L,119.9,66d3f78571f83874df5f631012ffe2b9
Lung_Opacity-4629.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4629.png,Lung_Opacity/images/Lung_Opacity-4629.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4629.png,Lung_Opacity/masks/Lung_Opacity-4629.png,True,False,299,299,L,123.07,4f3c7b1481028d31171be466f8460c58
Lung_Opacity-463.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-463.png,Lung_Opacity/images/Lung_Opacity-463.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-463.png,Lung_Opacity/masks/Lung_Opacity-463.png,True,False,299,299,L,122.49,c4618f7e2fa3bf5757b9b91213cd03b9
Lung_Opacity-4630.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4630.png,Lung_Opacity/images/Lung_Opacity-4630.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4630.png,Lung_Opacity/masks/Lung_Opacity-4630.png,True,False,299,299,L,151.91,cdf6b07c83052fccfe8ddd25fbfe6167
Lung_Opacity-4631.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4631.png,Lung_Opacity/images/Lung_Opacity-4631.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4631.png,Lung_Opacity/masks/Lung_Opacity-4631.png,True,False,299,299,L,129.4,542dad5ab31624b15f81f9e3390db8a2
Lung_Opacity-4632.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4632.png,Lung_Opacity/images/Lung_Opacity-4632.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4632.png,Lung_Opacity/masks/Lung_Opacity-4632.png,True,False,299,299,L,111.65,4218b534b14a327bd85778d67d7a3322
Lung_Opacity-4633.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4633.png,Lung_Opacity/images/Lung_Opacity-4633.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4633.png,Lung_Opacity/masks/Lung_Opacity-4633.png,True,False,299,299,L,160.19,65c43ed7bc7efbab2ed7a94364b3fff9
Lung_Opacity-4634.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4634.png,Lung_Opacity/images/Lung_Opacity-4634.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4634.png,Lung_Opacity/masks/Lung_Opacity-4634.png,True,False,299,299,L,169.32,a942d9bc6f72ae5930f2e5d8a036e4ff
Lung_Opacity-4635.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4635.png,Lung_Opacity/images/Lung_Opacity-4635.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4635.png,Lung_Opacity/masks/Lung_Opacity-4635.png,True,False,299,299,L,128.76,9ee55efdf870dfd9b8de341a669b45b2
Lung_Opacity-4636.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4636.png,Lung_Opacity/images/Lung_Opacity-4636.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4636.png,Lung_Opacity/masks/Lung_Opacity-4636.png,True,False,299,299,L,133.13,089c0965419aa4b5c605ac290f490c6a
Lung_Opacity-4637.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4637.png,Lung_Opacity/images/Lung_Opacity-4637.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4637.png,Lung_Opacity/masks/Lung_Opacity-4637.png,True,False,299,299,L,145.67,a25a0804cd7115ddc4d3cc09720a7fb6
Lung_Opacity-4638.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4638.png,Lung_Opacity/images/Lung_Opacity-4638.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4638.png,Lung_Opacity/masks/Lung_Opacity-4638.png,True,False,299,299,L,160.63,7d2549a9a38596c3ec16e353145e16e7
Lung_Opacity-4639.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4639.png,Lung_Opacity/images/Lung_Opacity-4639.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4639.png,Lung_Opacity/masks/Lung_Opacity-4639.png,True,False,299,299,L,153.06,585a77aa0bb0fa9448d31a26d59e850d
Lung_Opacity-464.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-464.png,Lung_Opacity/images/Lung_Opacity-464.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-464.png,Lung_Opacity/masks/Lung_Opacity-464.png,True,False,299,299,L,161.91,667f45921dac59f90e590eca149e39c0
Lung_Opacity-4640.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4640.png,Lung_Opacity/images/Lung_Opacity-4640.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4640.png,Lung_Opacity/masks/Lung_Opacity-4640.png,True,False,299,299,L,123.96,47dc864859bdfbec4968b86249ada540
Lung_Opacity-4641.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4641.png,Lung_Opacity/images/Lung_Opacity-4641.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4641.png,Lung_Opacity/masks/Lung_Opacity-4641.png,True,False,299,299,L,116.06,765d01c336afff6ff9628c7befdcbda8
Lung_Opacity-4642.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4642.png,Lung_Opacity/images/Lung_Opacity-4642.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4642.png,Lung_Opacity/masks/Lung_Opacity-4642.png,True,False,299,299,L,113.69,c18a7c755ca42c27044fa37ceb56f751
Lung_Opacity-4643.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4643.png,Lung_Opacity/images/Lung_Opacity-4643.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4643.png,Lung_Opacity/masks/Lung_Opacity-4643.png,True,False,299,299,L,158.47,2dabe58c64a51f7f5baa865a6f7ad682
Lung_Opacity-4644.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4644.png,Lung_Opacity/images/Lung_Opacity-4644.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4644.png,Lung_Opacity/masks/Lung_Opacity-4644.png,True,False,299,299,L,97.69,49b54634594d0f539ee2681494c945a0
Lung_Opacity-4645.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4645.png,Lung_Opacity/images/Lung_Opacity-4645.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4645.png,Lung_Opacity/masks/Lung_Opacity-4645.png,True,False,299,299,L,118.19,24eececb3315329d272b036f157efab0
Lung_Opacity-4646.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4646.png,Lung_Opacity/images/Lung_Opacity-4646.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4646.png,Lung_Opacity/masks/Lung_Opacity-4646.png,True,False,299,299,L,84.65,a1c952e12e3f643d8d09f5a03d89fad8
Lung_Opacity-4647.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4647.png,Lung_Opacity/images/Lung_Opacity-4647.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4647.png,Lung_Opacity/masks/Lung_Opacity-4647.png,True,False,299,299,L,106.81,87277a2613872aeb5df9ad1966678334
Lung_Opacity-4648.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4648.png,Lung_Opacity/images/Lung_Opacity-4648.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4648.png,Lung_Opacity/masks/Lung_Opacity-4648.png,True,False,299,299,L,143.68,ab3ad32393c486e4821cc7e311232e1e
Lung_Opacity-4649.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4649.png,Lung_Opacity/images/Lung_Opacity-4649.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4649.png,Lung_Opacity/masks/Lung_Opacity-4649.png,True,False,299,299,L,86.18,e99039ed71d12f803bdd4d959ffa6658
Lung_Opacity-465.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-465.png,Lung_Opacity/images/Lung_Opacity-465.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-465.png,Lung_Opacity/masks/Lung_Opacity-465.png,True,False,299,299,L,125.87,4ec3417d7b99c1697a7004d58b007f6a
Lung_Opacity-4650.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4650.png,Lung_Opacity/images/Lung_Opacity-4650.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4650.png,Lung_Opacity/masks/Lung_Opacity-4650.png,True,False,299,299,L,163.08,ba722e98a19abc1f433d93d9f5bc4ed3
Lung_Opacity-4651.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4651.png,Lung_Opacity/images/Lung_Opacity-4651.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4651.png,Lung_Opacity/masks/Lung_Opacity-4651.png,True,False,299,299,L,124.04,e36d5f04c0ad3a1dfe51b03654c11e19
Lung_Opacity-4652.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4652.png,Lung_Opacity/images/Lung_Opacity-4652.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4652.png,Lung_Opacity/masks/Lung_Opacity-4652.png,True,False,299,299,L,125.92,aa6ea9036ca5e1aa165b85bf13973af0
Lung_Opacity-4653.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4653.png,Lung_Opacity/images/Lung_Opacity-4653.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4653.png,Lung_Opacity/masks/Lung_Opacity-4653.png,True,False,299,299,L,195.61,436e28107ec722c8c6a57de3a436e6f1
Lung_Opacity-4654.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4654.png,Lung_Opacity/images/Lung_Opacity-4654.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4654.png,Lung_Opacity/masks/Lung_Opacity-4654.png,True,False,299,299,L,145.8,4f1f90263f246d5896c3be07a4a942fb
Lung_Opacity-4655.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4655.png,Lung_Opacity/images/Lung_Opacity-4655.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4655.png,Lung_Opacity/masks/Lung_Opacity-4655.png,True,False,299,299,L,118.74,4b578db07ac991902859ceac03b7f58b
Lung_Opacity-4656.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4656.png,Lung_Opacity/images/Lung_Opacity-4656.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4656.png,Lung_Opacity/masks/Lung_Opacity-4656.png,True,False,299,299,L,124.34,af13206bb6cc998756433b683389c74a
Lung_Opacity-4657.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4657.png,Lung_Opacity/images/Lung_Opacity-4657.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4657.png,Lung_Opacity/masks/Lung_Opacity-4657.png,True,False,299,299,L,114.24,c64ba54ef79ee6e211979c39e732b059
Lung_Opacity-4658.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4658.png,Lung_Opacity/images/Lung_Opacity-4658.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4658.png,Lung_Opacity/masks/Lung_Opacity-4658.png,True,False,299,299,L,185.09,e520b6f36d6e007d6fefcc5f18607b8a
Lung_Opacity-4659.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4659.png,Lung_Opacity/images/Lung_Opacity-4659.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4659.png,Lung_Opacity/masks/Lung_Opacity-4659.png,True,False,299,299,L,105.1,8e675e0a96cae529c2e4fc7c31294434
Lung_Opacity-466.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-466.png,Lung_Opacity/images/Lung_Opacity-466.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-466.png,Lung_Opacity/masks/Lung_Opacity-466.png,True,False,299,299,L,86.08,390030db3c35d5056bb668a42020194f
Lung_Opacity-4660.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4660.png,Lung_Opacity/images/Lung_Opacity-4660.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4660.png,Lung_Opacity/masks/Lung_Opacity-4660.png,True,False,299,299,L,128.31,1603f783defb28560b93d2456551b67d
Lung_Opacity-4661.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4661.png,Lung_Opacity/images/Lung_Opacity-4661.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4661.png,Lung_Opacity/masks/Lung_Opacity-4661.png,True,False,299,299,L,117.2,212e61e99bf2ef0484b213a223c38e28
Lung_Opacity-4662.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4662.png,Lung_Opacity/images/Lung_Opacity-4662.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4662.png,Lung_Opacity/masks/Lung_Opacity-4662.png,True,False,299,299,L,45.0,cb3fc3392e75d1e1b5feff2d801d5ae4
Lung_Opacity-4663.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4663.png,Lung_Opacity/images/Lung_Opacity-4663.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4663.png,Lung_Opacity/masks/Lung_Opacity-4663.png,True,False,299,299,L,105.14,d0eeee2923ffd0974e7eddce598627be
Lung_Opacity-4664.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4664.png,Lung_Opacity/images/Lung_Opacity-4664.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4664.png,Lung_Opacity/masks/Lung_Opacity-4664.png,True,False,299,299,L,122.21,d95af7aa10b351c2979afca203dece1b
Lung_Opacity-4665.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4665.png,Lung_Opacity/images/Lung_Opacity-4665.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4665.png,Lung_Opacity/masks/Lung_Opacity-4665.png,True,False,299,299,L,131.6,75c0e3718b8afa510ea4695771d8c9e9
Lung_Opacity-4666.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4666.png,Lung_Opacity/images/Lung_Opacity-4666.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4666.png,Lung_Opacity/masks/Lung_Opacity-4666.png,True,False,299,299,L,109.78,018568915cbaf4a697a8e4d43b14d0a3
Lung_Opacity-4667.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4667.png,Lung_Opacity/images/Lung_Opacity-4667.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4667.png,Lung_Opacity/masks/Lung_Opacity-4667.png,True,False,299,299,L,141.76,0223587d5cf6d3534c854fd91815c367
Lung_Opacity-4668.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4668.png,Lung_Opacity/images/Lung_Opacity-4668.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4668.png,Lung_Opacity/masks/Lung_Opacity-4668.png,True,False,299,299,L,141.93,cc09bbf21aa42bf2901d8a0055f922a8
Lung_Opacity-4669.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4669.png,Lung_Opacity/images/Lung_Opacity-4669.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4669.png,Lung_Opacity/masks/Lung_Opacity-4669.png,True,False,299,299,L,141.27,c46b2a7c92f1c7887fd3d2cb89ee9f14
Lung_Opacity-467.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-467.png,Lung_Opacity/images/Lung_Opacity-467.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-467.png,Lung_Opacity/masks/Lung_Opacity-467.png,True,False,299,299,L,173.38,c10e36f268aa5f8959e03fd9790266ff
Lung_Opacity-4670.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4670.png,Lung_Opacity/images/Lung_Opacity-4670.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4670.png,Lung_Opacity/masks/Lung_Opacity-4670.png,True,False,299,299,L,117.68,d7656d164d24e7448b51839f233d12ed
Lung_Opacity-4671.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4671.png,Lung_Opacity/images/Lung_Opacity-4671.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4671.png,Lung_Opacity/masks/Lung_Opacity-4671.png,True,False,299,299,L,102.5,274505073acda8437e7d51ebc6aa886b
Lung_Opacity-4672.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4672.png,Lung_Opacity/images/Lung_Opacity-4672.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4672.png,Lung_Opacity/masks/Lung_Opacity-4672.png,True,False,299,299,L,111.48,08ca2617dc199e4b19afb5eff5be57ba
Lung_Opacity-4673.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4673.png,Lung_Opacity/images/Lung_Opacity-4673.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4673.png,Lung_Opacity/masks/Lung_Opacity-4673.png,True,False,299,299,L,120.82,715849ee8cda8e5205bba1aa5af54dbb
Lung_Opacity-4674.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4674.png,Lung_Opacity/images/Lung_Opacity-4674.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4674.png,Lung_Opacity/masks/Lung_Opacity-4674.png,True,False,299,299,L,166.23,528e3b613e748a40241f12341a64c04d
Lung_Opacity-4675.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4675.png,Lung_Opacity/images/Lung_Opacity-4675.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4675.png,Lung_Opacity/masks/Lung_Opacity-4675.png,True,False,299,299,L,149.64,2abd7b232a41c329989d61c8e623154a
Lung_Opacity-4676.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4676.png,Lung_Opacity/images/Lung_Opacity-4676.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4676.png,Lung_Opacity/masks/Lung_Opacity-4676.png,True,False,299,299,L,118.68,b8c40b0d10574682b31c95cdfbc7b7ee
Lung_Opacity-4677.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4677.png,Lung_Opacity/images/Lung_Opacity-4677.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4677.png,Lung_Opacity/masks/Lung_Opacity-4677.png,True,False,299,299,L,121.34,2489b0912ada6155d5f9c1c5f1662bd5
Lung_Opacity-4678.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4678.png,Lung_Opacity/images/Lung_Opacity-4678.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4678.png,Lung_Opacity/masks/Lung_Opacity-4678.png,True,False,299,299,L,122.82,e52d42dc7184fbafe2da04c26e919444
Lung_Opacity-4679.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4679.png,Lung_Opacity/images/Lung_Opacity-4679.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4679.png,Lung_Opacity/masks/Lung_Opacity-4679.png,True,False,299,299,L,140.0,25ca22165b83785c03665c5dfd532821
Lung_Opacity-468.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-468.png,Lung_Opacity/images/Lung_Opacity-468.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-468.png,Lung_Opacity/masks/Lung_Opacity-468.png,True,False,299,299,L,128.98,91c2a64b6488328787f850527658abb9
Lung_Opacity-4680.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4680.png,Lung_Opacity/images/Lung_Opacity-4680.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4680.png,Lung_Opacity/masks/Lung_Opacity-4680.png,True,False,299,299,L,155.95,ab82d24f551bfb762b53cdc1b5417c9f
Lung_Opacity-4681.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4681.png,Lung_Opacity/images/Lung_Opacity-4681.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4681.png,Lung_Opacity/masks/Lung_Opacity-4681.png,True,False,299,299,L,124.43,9b6409951cd0bce36d84a2d4d2075cb4
Lung_Opacity-4682.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4682.png,Lung_Opacity/images/Lung_Opacity-4682.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4682.png,Lung_Opacity/masks/Lung_Opacity-4682.png,True,False,299,299,L,105.77,c6dd43c3d155378495fc9100d9c11565
Lung_Opacity-4683.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4683.png,Lung_Opacity/images/Lung_Opacity-4683.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4683.png,Lung_Opacity/masks/Lung_Opacity-4683.png,True,False,299,299,L,156.32,7cd92d2e5193a7d3c7181aaba1dbd454
Lung_Opacity-4684.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4684.png,Lung_Opacity/images/Lung_Opacity-4684.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4684.png,Lung_Opacity/masks/Lung_Opacity-4684.png,True,False,299,299,L,116.28,cf8666fe1a0469267c40a59179233b13
Lung_Opacity-4685.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4685.png,Lung_Opacity/images/Lung_Opacity-4685.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4685.png,Lung_Opacity/masks/Lung_Opacity-4685.png,True,False,299,299,L,158.5,edc7bb6772011f3cbebe5c19e96773e6
Lung_Opacity-4686.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4686.png,Lung_Opacity/images/Lung_Opacity-4686.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4686.png,Lung_Opacity/masks/Lung_Opacity-4686.png,True,False,299,299,L,119.59,5cdbe5e9ee995213d6eed4e61d447d59
Lung_Opacity-4687.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4687.png,Lung_Opacity/images/Lung_Opacity-4687.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4687.png,Lung_Opacity/masks/Lung_Opacity-4687.png,True,False,299,299,L,200.44,671102553549ef51b7a4c7827e965843
Lung_Opacity-4688.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4688.png,Lung_Opacity/images/Lung_Opacity-4688.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4688.png,Lung_Opacity/masks/Lung_Opacity-4688.png,True,False,299,299,L,110.28,f32550528e4842551221ff08d20bd416
Lung_Opacity-4689.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4689.png,Lung_Opacity/images/Lung_Opacity-4689.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4689.png,Lung_Opacity/masks/Lung_Opacity-4689.png,True,False,299,299,L,112.83,8d429811fb3385095d63dc82da7172d2
Lung_Opacity-469.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-469.png,Lung_Opacity/images/Lung_Opacity-469.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-469.png,Lung_Opacity/masks/Lung_Opacity-469.png,True,False,299,299,L,105.91,3edd7f8c445daac5e943949892b5f6d2
Lung_Opacity-4690.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4690.png,Lung_Opacity/images/Lung_Opacity-4690.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4690.png,Lung_Opacity/masks/Lung_Opacity-4690.png,True,False,299,299,L,98.32,2399c613cfe5bad0cfcaa5bc22c7655b
Lung_Opacity-4691.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4691.png,Lung_Opacity/images/Lung_Opacity-4691.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4691.png,Lung_Opacity/masks/Lung_Opacity-4691.png,True,False,299,299,L,135.51,3629b5ec537e4889e971a120680724d1
Lung_Opacity-4692.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4692.png,Lung_Opacity/images/Lung_Opacity-4692.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4692.png,Lung_Opacity/masks/Lung_Opacity-4692.png,True,False,299,299,L,123.25,ac0b0f4b04bac4e6296a8568edb7e3fc
Lung_Opacity-4693.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4693.png,Lung_Opacity/images/Lung_Opacity-4693.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4693.png,Lung_Opacity/masks/Lung_Opacity-4693.png,True,False,299,299,L,162.91,fdf3da2bcd0ae431e87260c6d51eabb4
Lung_Opacity-4694.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4694.png,Lung_Opacity/images/Lung_Opacity-4694.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4694.png,Lung_Opacity/masks/Lung_Opacity-4694.png,True,False,299,299,L,141.99,185d294755b4fc8fd23f253ef2c39453
Lung_Opacity-4695.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4695.png,Lung_Opacity/images/Lung_Opacity-4695.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4695.png,Lung_Opacity/masks/Lung_Opacity-4695.png,True,False,299,299,L,110.25,2a7038b2de53ba2d169a30f1d9bdae28
Lung_Opacity-4696.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4696.png,Lung_Opacity/images/Lung_Opacity-4696.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4696.png,Lung_Opacity/masks/Lung_Opacity-4696.png,True,False,299,299,L,113.51,532285019bdbc8596d4a75b289f0badd
Lung_Opacity-4697.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4697.png,Lung_Opacity/images/Lung_Opacity-4697.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4697.png,Lung_Opacity/masks/Lung_Opacity-4697.png,True,False,299,299,L,112.43,1f312a295f8936a30aa8de80c2cd51f5
Lung_Opacity-4698.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4698.png,Lung_Opacity/images/Lung_Opacity-4698.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4698.png,Lung_Opacity/masks/Lung_Opacity-4698.png,True,False,299,299,L,170.17,1a15bcb6d6f6ecdff30874bdb6bf7aa8
Lung_Opacity-4699.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4699.png,Lung_Opacity/images/Lung_Opacity-4699.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4699.png,Lung_Opacity/masks/Lung_Opacity-4699.png,True,False,299,299,L,172.91,0dc1054802da597f63f26f009446db92
Lung_Opacity-47.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-47.png,Lung_Opacity/images/Lung_Opacity-47.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-47.png,Lung_Opacity/masks/Lung_Opacity-47.png,True,False,299,299,L,141.01,aef627d5a3925525eb08d218c20d4325
Lung_Opacity-470.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-470.png,Lung_Opacity/images/Lung_Opacity-470.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-470.png,Lung_Opacity/masks/Lung_Opacity-470.png,True,False,299,299,L,171.82,9e380bb71da8c551a310f4ccbf600228
Lung_Opacity-4700.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4700.png,Lung_Opacity/images/Lung_Opacity-4700.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4700.png,Lung_Opacity/masks/Lung_Opacity-4700.png,True,False,299,299,L,125.16,bcd277e7d2412c4b46d5c8f44295cfa4
Lung_Opacity-4701.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4701.png,Lung_Opacity/images/Lung_Opacity-4701.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4701.png,Lung_Opacity/masks/Lung_Opacity-4701.png,True,False,299,299,L,121.87,4561b9c0213fc807e0da035808ccc9d3
Lung_Opacity-4702.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4702.png,Lung_Opacity/images/Lung_Opacity-4702.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4702.png,Lung_Opacity/masks/Lung_Opacity-4702.png,True,False,299,299,L,96.13,41079d3f753be763e02177fbe27bde24
Lung_Opacity-4703.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4703.png,Lung_Opacity/images/Lung_Opacity-4703.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4703.png,Lung_Opacity/masks/Lung_Opacity-4703.png,True,False,299,299,L,171.77,a942dac9106e37ce8844d485edde8b6d
Lung_Opacity-4704.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4704.png,Lung_Opacity/images/Lung_Opacity-4704.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4704.png,Lung_Opacity/masks/Lung_Opacity-4704.png,True,False,299,299,L,109.31,8c01cf9381563d2cc074b8f6e8381de8
Lung_Opacity-4705.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4705.png,Lung_Opacity/images/Lung_Opacity-4705.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4705.png,Lung_Opacity/masks/Lung_Opacity-4705.png,True,False,299,299,L,103.99,6d9068c9c32774c2eb1420896ba9b860
Lung_Opacity-4706.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4706.png,Lung_Opacity/images/Lung_Opacity-4706.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4706.png,Lung_Opacity/masks/Lung_Opacity-4706.png,True,False,299,299,L,111.01,ce1c5ced647f98b6cc0a20b12d12411c
Lung_Opacity-4707.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4707.png,Lung_Opacity/images/Lung_Opacity-4707.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4707.png,Lung_Opacity/masks/Lung_Opacity-4707.png,True,False,299,299,L,103.13,1f8aab1fc6f16a5f299b610468bc88d9
Lung_Opacity-4708.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4708.png,Lung_Opacity/images/Lung_Opacity-4708.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4708.png,Lung_Opacity/masks/Lung_Opacity-4708.png,True,False,299,299,L,126.85,28972344efe9ec44ba81959a79b03bc3
Lung_Opacity-4709.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4709.png,Lung_Opacity/images/Lung_Opacity-4709.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4709.png,Lung_Opacity/masks/Lung_Opacity-4709.png,True,False,299,299,L,117.56,68b8b38def52d60ffdbfcc05e4f6ab40
Lung_Opacity-471.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-471.png,Lung_Opacity/images/Lung_Opacity-471.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-471.png,Lung_Opacity/masks/Lung_Opacity-471.png,True,False,299,299,L,161.94,166ad6aa4855897f92bf108e01bbd3c9
Lung_Opacity-4710.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4710.png,Lung_Opacity/images/Lung_Opacity-4710.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4710.png,Lung_Opacity/masks/Lung_Opacity-4710.png,True,False,299,299,L,92.78,ff8b62eb5470efeabfa30720829e5c33
Lung_Opacity-4711.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4711.png,Lung_Opacity/images/Lung_Opacity-4711.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4711.png,Lung_Opacity/masks/Lung_Opacity-4711.png,True,False,299,299,L,141.73,720f1288d2f5e552e760b1049f8a1e18
Lung_Opacity-4712.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4712.png,Lung_Opacity/images/Lung_Opacity-4712.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4712.png,Lung_Opacity/masks/Lung_Opacity-4712.png,True,False,299,299,L,104.94,afe6be42af8b905e10ecb19c0590bf8a
Lung_Opacity-4713.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4713.png,Lung_Opacity/images/Lung_Opacity-4713.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4713.png,Lung_Opacity/masks/Lung_Opacity-4713.png,True,False,299,299,L,140.47,2366dde57f960c2f3fd013026deef431
Lung_Opacity-4714.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4714.png,Lung_Opacity/images/Lung_Opacity-4714.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4714.png,Lung_Opacity/masks/Lung_Opacity-4714.png,True,False,299,299,L,111.36,bf45634a1675d7d0bb7347494460f635
Lung_Opacity-4715.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4715.png,Lung_Opacity/images/Lung_Opacity-4715.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4715.png,Lung_Opacity/masks/Lung_Opacity-4715.png,True,False,299,299,L,138.76,641e1a3ca2065fd2320586c5219fc988
Lung_Opacity-4716.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4716.png,Lung_Opacity/images/Lung_Opacity-4716.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4716.png,Lung_Opacity/masks/Lung_Opacity-4716.png,True,False,299,299,L,111.59,55a79531225b771c69bb70cfc07edfa1
Lung_Opacity-4717.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4717.png,Lung_Opacity/images/Lung_Opacity-4717.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4717.png,Lung_Opacity/masks/Lung_Opacity-4717.png,True,False,299,299,L,70.1,dbbfb7d52086d52eb246fe27d6f76254
Lung_Opacity-4718.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4718.png,Lung_Opacity/images/Lung_Opacity-4718.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4718.png,Lung_Opacity/masks/Lung_Opacity-4718.png,True,False,299,299,L,157.64,a85367f8a975a23e1beedd0a284b9acf
Lung_Opacity-4719.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4719.png,Lung_Opacity/images/Lung_Opacity-4719.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4719.png,Lung_Opacity/masks/Lung_Opacity-4719.png,True,False,299,299,L,104.51,8aca4e27b2fcf540fccdc02bb67d1af0
Lung_Opacity-472.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-472.png,Lung_Opacity/images/Lung_Opacity-472.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-472.png,Lung_Opacity/masks/Lung_Opacity-472.png,True,False,299,299,L,117.55,8b9f0224ce7c7696afd30484f3bc43ea
Lung_Opacity-4720.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4720.png,Lung_Opacity/images/Lung_Opacity-4720.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4720.png,Lung_Opacity/masks/Lung_Opacity-4720.png,True,False,299,299,L,96.3,71b4c357828d245e09f79bda560c90f0
Lung_Opacity-4721.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4721.png,Lung_Opacity/images/Lung_Opacity-4721.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4721.png,Lung_Opacity/masks/Lung_Opacity-4721.png,True,False,299,299,L,116.04,4be5a67fc2f15e94f04ca819fbfb1d72
Lung_Opacity-4722.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4722.png,Lung_Opacity/images/Lung_Opacity-4722.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4722.png,Lung_Opacity/masks/Lung_Opacity-4722.png,True,False,299,299,L,171.21,030864c0dd72d7fe453c05e0df9e1714
Lung_Opacity-4723.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4723.png,Lung_Opacity/images/Lung_Opacity-4723.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4723.png,Lung_Opacity/masks/Lung_Opacity-4723.png,True,False,299,299,L,119.81,22b77396f371515dc5cd26a08273a82d
Lung_Opacity-4724.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4724.png,Lung_Opacity/images/Lung_Opacity-4724.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4724.png,Lung_Opacity/masks/Lung_Opacity-4724.png,True,False,299,299,L,117.77,54f2c8d8b000520cf37f34f178928514
Lung_Opacity-4725.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4725.png,Lung_Opacity/images/Lung_Opacity-4725.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4725.png,Lung_Opacity/masks/Lung_Opacity-4725.png,True,False,299,299,L,124.79,2ee4665ec2f3f56830ae87b26aa7254e
Lung_Opacity-4726.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4726.png,Lung_Opacity/images/Lung_Opacity-4726.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4726.png,Lung_Opacity/masks/Lung_Opacity-4726.png,True,False,299,299,L,105.99,1b1b310fae47f05df5e7387207dfc02d
Lung_Opacity-4727.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4727.png,Lung_Opacity/images/Lung_Opacity-4727.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4727.png,Lung_Opacity/masks/Lung_Opacity-4727.png,True,False,299,299,L,181.31,7a29247bd21c95777f34ff87bd6c0df2
Lung_Opacity-4728.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4728.png,Lung_Opacity/images/Lung_Opacity-4728.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4728.png,Lung_Opacity/masks/Lung_Opacity-4728.png,True,False,299,299,L,135.45,b7d309ad9c2e9284d7fcf0231cf02dbb
Lung_Opacity-4729.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4729.png,Lung_Opacity/images/Lung_Opacity-4729.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4729.png,Lung_Opacity/masks/Lung_Opacity-4729.png,True,False,299,299,L,94.12,cdbb4ffa6b531086b9e323eb299d8d2c
Lung_Opacity-473.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-473.png,Lung_Opacity/images/Lung_Opacity-473.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-473.png,Lung_Opacity/masks/Lung_Opacity-473.png,True,False,299,299,L,137.22,638e75a199e9ab2785e5974820c7f1fe
Lung_Opacity-4730.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4730.png,Lung_Opacity/images/Lung_Opacity-4730.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4730.png,Lung_Opacity/masks/Lung_Opacity-4730.png,True,False,299,299,L,132.06,1fe9debb32e390d5f1dadd678e092bd3
Lung_Opacity-4731.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4731.png,Lung_Opacity/images/Lung_Opacity-4731.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4731.png,Lung_Opacity/masks/Lung_Opacity-4731.png,True,False,299,299,L,140.19,0cb8002a97ba5074bccd7bfc221c959c
Lung_Opacity-4732.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4732.png,Lung_Opacity/images/Lung_Opacity-4732.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4732.png,Lung_Opacity/masks/Lung_Opacity-4732.png,True,False,299,299,L,144.12,03200c1ccf50ab11051ccba2a2e148a0
Lung_Opacity-4733.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4733.png,Lung_Opacity/images/Lung_Opacity-4733.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4733.png,Lung_Opacity/masks/Lung_Opacity-4733.png,True,False,299,299,L,100.78,ad2bc671840fdccd76b2e8d29276dbb3
Lung_Opacity-4734.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4734.png,Lung_Opacity/images/Lung_Opacity-4734.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4734.png,Lung_Opacity/masks/Lung_Opacity-4734.png,True,False,299,299,L,124.83,07de20dcd219cc4f9495e77e7b9eb9ff
Lung_Opacity-4735.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4735.png,Lung_Opacity/images/Lung_Opacity-4735.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4735.png,Lung_Opacity/masks/Lung_Opacity-4735.png,True,False,299,299,L,150.79,c0d4ba6ad7082938bc1547ac85b4886b
Lung_Opacity-4736.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4736.png,Lung_Opacity/images/Lung_Opacity-4736.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4736.png,Lung_Opacity/masks/Lung_Opacity-4736.png,True,False,299,299,L,106.21,8c9d291f8f14a627cd4ea9ad6ed14be4
Lung_Opacity-4737.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4737.png,Lung_Opacity/images/Lung_Opacity-4737.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4737.png,Lung_Opacity/masks/Lung_Opacity-4737.png,True,False,299,299,L,116.01,b0743241d19c6f4d8b7d5f2624502fef
Lung_Opacity-4738.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4738.png,Lung_Opacity/images/Lung_Opacity-4738.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4738.png,Lung_Opacity/masks/Lung_Opacity-4738.png,True,False,299,299,L,125.23,a3808ada1ec0da2c656f494b8728b348
Lung_Opacity-4739.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4739.png,Lung_Opacity/images/Lung_Opacity-4739.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4739.png,Lung_Opacity/masks/Lung_Opacity-4739.png,True,False,299,299,L,108.26,bc070642451bc39e1c727fb841972ed5
Lung_Opacity-474.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-474.png,Lung_Opacity/images/Lung_Opacity-474.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-474.png,Lung_Opacity/masks/Lung_Opacity-474.png,True,False,299,299,L,111.07,6ebe0229fc31d86f6ea149ea80382ff2
Lung_Opacity-4740.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4740.png,Lung_Opacity/images/Lung_Opacity-4740.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4740.png,Lung_Opacity/masks/Lung_Opacity-4740.png,True,False,299,299,L,110.36,b2c4fd644b073a2cd6b2576e618c3961
Lung_Opacity-4741.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4741.png,Lung_Opacity/images/Lung_Opacity-4741.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4741.png,Lung_Opacity/masks/Lung_Opacity-4741.png,True,False,299,299,L,120.67,ad846abf61c489341fda5f262b3c00d1
Lung_Opacity-4742.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4742.png,Lung_Opacity/images/Lung_Opacity-4742.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4742.png,Lung_Opacity/masks/Lung_Opacity-4742.png,True,False,299,299,L,172.6,72b237fc2345126d0fdb54e74949b01c
Lung_Opacity-4743.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4743.png,Lung_Opacity/images/Lung_Opacity-4743.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4743.png,Lung_Opacity/masks/Lung_Opacity-4743.png,True,False,299,299,L,106.97,b8d5af8779a4b31ca97f41c46ebc206b
Lung_Opacity-4744.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4744.png,Lung_Opacity/images/Lung_Opacity-4744.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4744.png,Lung_Opacity/masks/Lung_Opacity-4744.png,True,False,299,299,L,152.75,4585c1e7054e4b323cbd26e5efe1a241
Lung_Opacity-4745.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4745.png,Lung_Opacity/images/Lung_Opacity-4745.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4745.png,Lung_Opacity/masks/Lung_Opacity-4745.png,True,False,299,299,L,116.28,defb059dc490b3e8aa4625f2808fb9cb
Lung_Opacity-4746.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4746.png,Lung_Opacity/images/Lung_Opacity-4746.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4746.png,Lung_Opacity/masks/Lung_Opacity-4746.png,True,False,299,299,L,127.0,1ce56fb2c34e71b02198e75db1b26d0b
Lung_Opacity-4747.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4747.png,Lung_Opacity/images/Lung_Opacity-4747.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4747.png,Lung_Opacity/masks/Lung_Opacity-4747.png,True,False,299,299,L,92.59,87235221b126d8395474f2c2931439e1
Lung_Opacity-4748.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4748.png,Lung_Opacity/images/Lung_Opacity-4748.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4748.png,Lung_Opacity/masks/Lung_Opacity-4748.png,True,False,299,299,L,110.82,e52030811086be61eff83cd0949bdd8c
Lung_Opacity-4749.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4749.png,Lung_Opacity/images/Lung_Opacity-4749.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4749.png,Lung_Opacity/masks/Lung_Opacity-4749.png,True,False,299,299,L,98.23,ee6ce611bbaf50d4c81aaf8acd344fbf
Lung_Opacity-475.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-475.png,Lung_Opacity/images/Lung_Opacity-475.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-475.png,Lung_Opacity/masks/Lung_Opacity-475.png,True,False,299,299,L,133.84,f926f4e974945bf030a2dbf41f112f98
Lung_Opacity-4750.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4750.png,Lung_Opacity/images/Lung_Opacity-4750.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4750.png,Lung_Opacity/masks/Lung_Opacity-4750.png,True,False,299,299,L,148.97,6b725fc8679a32970ea57ea0da35b79f
Lung_Opacity-4751.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4751.png,Lung_Opacity/images/Lung_Opacity-4751.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4751.png,Lung_Opacity/masks/Lung_Opacity-4751.png,True,False,299,299,L,120.71,36b3a8a87c4aa039b7f30eb51fabff4a
Lung_Opacity-4752.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4752.png,Lung_Opacity/images/Lung_Opacity-4752.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4752.png,Lung_Opacity/masks/Lung_Opacity-4752.png,True,False,299,299,L,158.0,4d39d8f4c6751e617dcac39f25506fe4
Lung_Opacity-4753.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4753.png,Lung_Opacity/images/Lung_Opacity-4753.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4753.png,Lung_Opacity/masks/Lung_Opacity-4753.png,True,False,299,299,L,91.96,ea892853baa0cfd15aff79402e360f33
Lung_Opacity-4754.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4754.png,Lung_Opacity/images/Lung_Opacity-4754.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4754.png,Lung_Opacity/masks/Lung_Opacity-4754.png,True,False,299,299,L,153.41,20880f36f3fa96ffc3fcfd711f914fcf
Lung_Opacity-4755.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4755.png,Lung_Opacity/images/Lung_Opacity-4755.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4755.png,Lung_Opacity/masks/Lung_Opacity-4755.png,True,False,299,299,L,111.12,54d75a99fe65b44938b2fb7857452b36
Lung_Opacity-4756.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4756.png,Lung_Opacity/images/Lung_Opacity-4756.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4756.png,Lung_Opacity/masks/Lung_Opacity-4756.png,True,False,299,299,L,110.48,d8549b55b30d68a0944fdd1b318ec516
Lung_Opacity-4757.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4757.png,Lung_Opacity/images/Lung_Opacity-4757.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4757.png,Lung_Opacity/masks/Lung_Opacity-4757.png,True,False,299,299,L,164.0,04b13e43f911753a9c080a7c5ac5bbd6
Lung_Opacity-4758.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4758.png,Lung_Opacity/images/Lung_Opacity-4758.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4758.png,Lung_Opacity/masks/Lung_Opacity-4758.png,True,False,299,299,L,117.28,cb36413bc87e5eaff0c92a5c76ef650e
Lung_Opacity-4759.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4759.png,Lung_Opacity/images/Lung_Opacity-4759.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4759.png,Lung_Opacity/masks/Lung_Opacity-4759.png,True,False,299,299,L,162.48,d36a58751a0ce6dfc69a01526a8193f9
Lung_Opacity-476.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-476.png,Lung_Opacity/images/Lung_Opacity-476.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-476.png,Lung_Opacity/masks/Lung_Opacity-476.png,True,False,299,299,L,120.66,282b6ab1b9f9c881e3f4c1414f762985
Lung_Opacity-4760.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4760.png,Lung_Opacity/images/Lung_Opacity-4760.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4760.png,Lung_Opacity/masks/Lung_Opacity-4760.png,True,False,299,299,L,125.49,3bc89d0459a9490f65dbf8f472900267
Lung_Opacity-4761.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4761.png,Lung_Opacity/images/Lung_Opacity-4761.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4761.png,Lung_Opacity/masks/Lung_Opacity-4761.png,True,False,299,299,L,117.58,567cbc9c947863aa515c0ccf1519012e
Lung_Opacity-4762.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4762.png,Lung_Opacity/images/Lung_Opacity-4762.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4762.png,Lung_Opacity/masks/Lung_Opacity-4762.png,True,False,299,299,L,150.04,9147a717de2ab998bbb90a217dde57f2
Lung_Opacity-4763.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4763.png,Lung_Opacity/images/Lung_Opacity-4763.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4763.png,Lung_Opacity/masks/Lung_Opacity-4763.png,True,False,299,299,L,131.36,153ea2f46e4d1aabf8a57802432453c4
Lung_Opacity-4764.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4764.png,Lung_Opacity/images/Lung_Opacity-4764.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4764.png,Lung_Opacity/masks/Lung_Opacity-4764.png,True,False,299,299,L,127.06,fc00068864c6230d68dfeff6cf5589d5
Lung_Opacity-4765.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4765.png,Lung_Opacity/images/Lung_Opacity-4765.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4765.png,Lung_Opacity/masks/Lung_Opacity-4765.png,True,False,299,299,L,168.7,d22c36a28bd16f47ebcdc22614144fac
Lung_Opacity-4766.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4766.png,Lung_Opacity/images/Lung_Opacity-4766.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4766.png,Lung_Opacity/masks/Lung_Opacity-4766.png,True,False,299,299,L,146.2,2bf77f0296e482a109a2b8620daa68fa
Lung_Opacity-4767.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4767.png,Lung_Opacity/images/Lung_Opacity-4767.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4767.png,Lung_Opacity/masks/Lung_Opacity-4767.png,True,False,299,299,L,111.37,492e2e731a4c80922b1c86ff684c0f6c
Lung_Opacity-4768.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4768.png,Lung_Opacity/images/Lung_Opacity-4768.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4768.png,Lung_Opacity/masks/Lung_Opacity-4768.png,True,False,299,299,L,125.47,8392326efca8ab6b69a6588587036c12
Lung_Opacity-4769.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4769.png,Lung_Opacity/images/Lung_Opacity-4769.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4769.png,Lung_Opacity/masks/Lung_Opacity-4769.png,True,False,299,299,L,104.07,282ec4ba76362596a1ec4b7ab938bdc3
Lung_Opacity-477.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-477.png,Lung_Opacity/images/Lung_Opacity-477.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-477.png,Lung_Opacity/masks/Lung_Opacity-477.png,True,False,299,299,L,147.84,a88141d8a6de74b456f50386b6f6b571
Lung_Opacity-4770.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4770.png,Lung_Opacity/images/Lung_Opacity-4770.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4770.png,Lung_Opacity/masks/Lung_Opacity-4770.png,True,False,299,299,L,107.53,e3e7b15568c4d2c171dd3c909b748a20
Lung_Opacity-4771.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4771.png,Lung_Opacity/images/Lung_Opacity-4771.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4771.png,Lung_Opacity/masks/Lung_Opacity-4771.png,True,False,299,299,L,108.39,148765a44a90910a56a85a13802ed6d5
Lung_Opacity-4772.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4772.png,Lung_Opacity/images/Lung_Opacity-4772.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4772.png,Lung_Opacity/masks/Lung_Opacity-4772.png,True,False,299,299,L,94.93,d339dea80eda6d925013450c60d22274
Lung_Opacity-4773.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4773.png,Lung_Opacity/images/Lung_Opacity-4773.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4773.png,Lung_Opacity/masks/Lung_Opacity-4773.png,True,False,299,299,L,155.89,ff7ed566cf020a4bfbd999f100a2b1a5
Lung_Opacity-4774.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4774.png,Lung_Opacity/images/Lung_Opacity-4774.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4774.png,Lung_Opacity/masks/Lung_Opacity-4774.png,True,False,299,299,L,101.96,53881aec23cd0037d178f584675227a7
Lung_Opacity-4775.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4775.png,Lung_Opacity/images/Lung_Opacity-4775.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4775.png,Lung_Opacity/masks/Lung_Opacity-4775.png,True,False,299,299,L,114.12,c37673b44ad03cabc89666e049618de2
Lung_Opacity-4776.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4776.png,Lung_Opacity/images/Lung_Opacity-4776.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4776.png,Lung_Opacity/masks/Lung_Opacity-4776.png,True,False,299,299,L,146.82,a334bc495ada3bfe7e0f2293f86ea0bb
Lung_Opacity-4777.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4777.png,Lung_Opacity/images/Lung_Opacity-4777.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4777.png,Lung_Opacity/masks/Lung_Opacity-4777.png,True,False,299,299,L,107.09,4068aa8907c0a0b7e7d30173f7804f6b
Lung_Opacity-4778.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4778.png,Lung_Opacity/images/Lung_Opacity-4778.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4778.png,Lung_Opacity/masks/Lung_Opacity-4778.png,True,False,299,299,L,136.65,1097bbe6b52b061d27965df23784d6a6
Lung_Opacity-4779.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4779.png,Lung_Opacity/images/Lung_Opacity-4779.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4779.png,Lung_Opacity/masks/Lung_Opacity-4779.png,True,False,299,299,L,115.49,9c032b96a95e2e8e98f3ecf5075a0be6
Lung_Opacity-478.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-478.png,Lung_Opacity/images/Lung_Opacity-478.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-478.png,Lung_Opacity/masks/Lung_Opacity-478.png,True,False,299,299,L,113.18,f621a2e6727ced0ce6ab061fba041ea3
Lung_Opacity-4780.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4780.png,Lung_Opacity/images/Lung_Opacity-4780.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4780.png,Lung_Opacity/masks/Lung_Opacity-4780.png,True,False,299,299,L,103.06,0fe863bc816dbaaeb3776ba3ec6bbeb4
Lung_Opacity-4781.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4781.png,Lung_Opacity/images/Lung_Opacity-4781.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4781.png,Lung_Opacity/masks/Lung_Opacity-4781.png,True,False,299,299,L,105.31,752a6616ac1bda8c87d36b5ccabba109
Lung_Opacity-4782.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4782.png,Lung_Opacity/images/Lung_Opacity-4782.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4782.png,Lung_Opacity/masks/Lung_Opacity-4782.png,True,False,299,299,L,111.49,adb28eb2d234b8ec65b86daffe5f15b3
Lung_Opacity-4783.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4783.png,Lung_Opacity/images/Lung_Opacity-4783.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4783.png,Lung_Opacity/masks/Lung_Opacity-4783.png,True,False,299,299,L,140.7,0be875c3629a62ccc1c20174afe70db1
Lung_Opacity-4784.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4784.png,Lung_Opacity/images/Lung_Opacity-4784.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4784.png,Lung_Opacity/masks/Lung_Opacity-4784.png,True,False,299,299,L,117.1,65e02accef0b082184f61cafc6c10da9
Lung_Opacity-4785.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4785.png,Lung_Opacity/images/Lung_Opacity-4785.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4785.png,Lung_Opacity/masks/Lung_Opacity-4785.png,True,False,299,299,L,120.18,e44f811b91fde2228afe335a7a5bef67
Lung_Opacity-4786.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4786.png,Lung_Opacity/images/Lung_Opacity-4786.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4786.png,Lung_Opacity/masks/Lung_Opacity-4786.png,True,False,299,299,L,116.52,7f499e23a3c317dd8e2fee6da1fd4baa
Lung_Opacity-4787.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4787.png,Lung_Opacity/images/Lung_Opacity-4787.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4787.png,Lung_Opacity/masks/Lung_Opacity-4787.png,True,False,299,299,L,153.13,444f15ae4936d5a3d075722386da1d4c
Lung_Opacity-4788.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4788.png,Lung_Opacity/images/Lung_Opacity-4788.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4788.png,Lung_Opacity/masks/Lung_Opacity-4788.png,True,False,299,299,L,115.82,31aec8dbebe863c589e89f98f6025763
Lung_Opacity-4789.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4789.png,Lung_Opacity/images/Lung_Opacity-4789.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4789.png,Lung_Opacity/masks/Lung_Opacity-4789.png,True,False,299,299,L,110.33,c1c13d4c2cc392fbce8dd0dc379ab7ac
Lung_Opacity-479.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-479.png,Lung_Opacity/images/Lung_Opacity-479.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-479.png,Lung_Opacity/masks/Lung_Opacity-479.png,True,False,299,299,L,123.34,322ef6f5431a8ecf14551d139b3d0685
Lung_Opacity-4790.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4790.png,Lung_Opacity/images/Lung_Opacity-4790.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4790.png,Lung_Opacity/masks/Lung_Opacity-4790.png,True,False,299,299,L,104.26,4728e8d586c99f5f53d01a499c434b04
Lung_Opacity-4791.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4791.png,Lung_Opacity/images/Lung_Opacity-4791.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4791.png,Lung_Opacity/masks/Lung_Opacity-4791.png,True,False,299,299,L,126.05,da979172aef331ba9b4e4b195d044248
Lung_Opacity-4792.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4792.png,Lung_Opacity/images/Lung_Opacity-4792.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4792.png,Lung_Opacity/masks/Lung_Opacity-4792.png,True,False,299,299,L,110.99,d37ec8abe708f2cf757ebdcbc5a772aa
Lung_Opacity-4793.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4793.png,Lung_Opacity/images/Lung_Opacity-4793.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4793.png,Lung_Opacity/masks/Lung_Opacity-4793.png,True,False,299,299,L,105.91,a222baf18cea22145128cd78225a3640
Lung_Opacity-4794.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4794.png,Lung_Opacity/images/Lung_Opacity-4794.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4794.png,Lung_Opacity/masks/Lung_Opacity-4794.png,True,False,299,299,L,147.92,82fcafb988179369e28c7ef64d18d29d
Lung_Opacity-4795.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4795.png,Lung_Opacity/images/Lung_Opacity-4795.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4795.png,Lung_Opacity/masks/Lung_Opacity-4795.png,True,False,299,299,L,116.7,5505df7099416e8ad7e2b46289d7d1a4
Lung_Opacity-4796.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4796.png,Lung_Opacity/images/Lung_Opacity-4796.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4796.png,Lung_Opacity/masks/Lung_Opacity-4796.png,True,False,299,299,L,132.35,48ca68695d36857124b4be03b1f82b92
Lung_Opacity-4797.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4797.png,Lung_Opacity/images/Lung_Opacity-4797.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4797.png,Lung_Opacity/masks/Lung_Opacity-4797.png,True,False,299,299,L,125.54,bbe6279b35894c90a5e457fc9184143a
Lung_Opacity-4798.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4798.png,Lung_Opacity/images/Lung_Opacity-4798.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4798.png,Lung_Opacity/masks/Lung_Opacity-4798.png,True,False,299,299,L,147.73,2fabd78b80c7204e169292eccff8b7dc
Lung_Opacity-4799.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4799.png,Lung_Opacity/images/Lung_Opacity-4799.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4799.png,Lung_Opacity/masks/Lung_Opacity-4799.png,True,False,299,299,L,167.75,e8e87a6d0efda5bc17e3afd76b2503ba
Lung_Opacity-48.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-48.png,Lung_Opacity/images/Lung_Opacity-48.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-48.png,Lung_Opacity/masks/Lung_Opacity-48.png,True,False,299,299,L,149.03,a0eba6dab53e6c73565d849511ac02a8
Lung_Opacity-480.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-480.png,Lung_Opacity/images/Lung_Opacity-480.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-480.png,Lung_Opacity/masks/Lung_Opacity-480.png,True,False,299,299,L,131.73,f3ac60651be9320d6c87c661135a61b3
Lung_Opacity-4800.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4800.png,Lung_Opacity/images/Lung_Opacity-4800.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4800.png,Lung_Opacity/masks/Lung_Opacity-4800.png,True,False,299,299,L,115.38,bf3cdaf28e4a6d5a570dc8d2bade38d7
Lung_Opacity-4801.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4801.png,Lung_Opacity/images/Lung_Opacity-4801.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4801.png,Lung_Opacity/masks/Lung_Opacity-4801.png,True,False,299,299,L,110.6,9433b68b3a9a4199992237ce29f6009c
Lung_Opacity-4802.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4802.png,Lung_Opacity/images/Lung_Opacity-4802.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4802.png,Lung_Opacity/masks/Lung_Opacity-4802.png,True,False,299,299,L,127.74,7e6e5374d4ddf66f2e15f2fc8d01dabb
Lung_Opacity-4803.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4803.png,Lung_Opacity/images/Lung_Opacity-4803.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4803.png,Lung_Opacity/masks/Lung_Opacity-4803.png,True,False,299,299,L,116.84,6b3cc93d4b6a22c9863b9ad62600db81
Lung_Opacity-4804.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4804.png,Lung_Opacity/images/Lung_Opacity-4804.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4804.png,Lung_Opacity/masks/Lung_Opacity-4804.png,True,False,299,299,L,104.26,bb833ff5e9ed608c113d3895ec1cfa4f
Lung_Opacity-4805.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4805.png,Lung_Opacity/images/Lung_Opacity-4805.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4805.png,Lung_Opacity/masks/Lung_Opacity-4805.png,True,False,299,299,L,148.64,a3c9ce273e404b2b8f69c9b81b82cf60
Lung_Opacity-4806.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4806.png,Lung_Opacity/images/Lung_Opacity-4806.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4806.png,Lung_Opacity/masks/Lung_Opacity-4806.png,True,False,299,299,L,150.1,a46b3535a65979652a2d3b4b46bfaf93
Lung_Opacity-4807.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4807.png,Lung_Opacity/images/Lung_Opacity-4807.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4807.png,Lung_Opacity/masks/Lung_Opacity-4807.png,True,False,299,299,L,154.62,d054949ece27a895afd22c93d1e94a0e
Lung_Opacity-4808.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4808.png,Lung_Opacity/images/Lung_Opacity-4808.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4808.png,Lung_Opacity/masks/Lung_Opacity-4808.png,True,False,299,299,L,114.68,ee61acb09ef6488e531719e4f31bc593
Lung_Opacity-4809.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4809.png,Lung_Opacity/images/Lung_Opacity-4809.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4809.png,Lung_Opacity/masks/Lung_Opacity-4809.png,True,False,299,299,L,105.15,f8ffcc2470db01e5f286a5cddb6cf1f4
Lung_Opacity-481.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-481.png,Lung_Opacity/images/Lung_Opacity-481.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-481.png,Lung_Opacity/masks/Lung_Opacity-481.png,True,False,299,299,L,115.24,65610a8499d549d472a1eae4a4f092e9
Lung_Opacity-4810.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4810.png,Lung_Opacity/images/Lung_Opacity-4810.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4810.png,Lung_Opacity/masks/Lung_Opacity-4810.png,True,False,299,299,L,121.93,fbc4545134dcfcea9c454f7f9482e8d5
Lung_Opacity-4811.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4811.png,Lung_Opacity/images/Lung_Opacity-4811.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4811.png,Lung_Opacity/masks/Lung_Opacity-4811.png,True,False,299,299,L,126.24,839cd2f8dbfcce78002dc375d001bbec
Lung_Opacity-4812.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4812.png,Lung_Opacity/images/Lung_Opacity-4812.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4812.png,Lung_Opacity/masks/Lung_Opacity-4812.png,True,False,299,299,L,165.01,49fd4286470d1fb3667ea398fb445b6e
Lung_Opacity-4813.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4813.png,Lung_Opacity/images/Lung_Opacity-4813.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4813.png,Lung_Opacity/masks/Lung_Opacity-4813.png,True,False,299,299,L,136.31,1e46b40610e6e43053191c9efbf681fc
Lung_Opacity-4814.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4814.png,Lung_Opacity/images/Lung_Opacity-4814.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4814.png,Lung_Opacity/masks/Lung_Opacity-4814.png,True,False,299,299,L,96.32,d3ea5e224c133adbc596fc13bbdb20d4
Lung_Opacity-4815.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4815.png,Lung_Opacity/images/Lung_Opacity-4815.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4815.png,Lung_Opacity/masks/Lung_Opacity-4815.png,True,False,299,299,L,118.92,9827262dcda15a2a33a4f8fc58c0b62f
Lung_Opacity-4816.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4816.png,Lung_Opacity/images/Lung_Opacity-4816.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4816.png,Lung_Opacity/masks/Lung_Opacity-4816.png,True,False,299,299,L,125.18,9dc425c07a533c9f8aec3efb34d8ae3a
Lung_Opacity-4817.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4817.png,Lung_Opacity/images/Lung_Opacity-4817.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4817.png,Lung_Opacity/masks/Lung_Opacity-4817.png,True,False,299,299,L,113.17,9222a5dddc5b0872702ce56f2f8c3550
Lung_Opacity-4818.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4818.png,Lung_Opacity/images/Lung_Opacity-4818.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4818.png,Lung_Opacity/masks/Lung_Opacity-4818.png,True,False,299,299,L,117.13,2cc4d9399317a2d2cb7f1167b4db63a2
Lung_Opacity-4819.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4819.png,Lung_Opacity/images/Lung_Opacity-4819.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4819.png,Lung_Opacity/masks/Lung_Opacity-4819.png,True,False,299,299,L,134.74,7041aecf184b1f1caeb42ec95672b7c7
Lung_Opacity-482.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-482.png,Lung_Opacity/images/Lung_Opacity-482.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-482.png,Lung_Opacity/masks/Lung_Opacity-482.png,True,False,299,299,L,82.89,8a048be45362f58e9e974eada1408402
Lung_Opacity-4820.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4820.png,Lung_Opacity/images/Lung_Opacity-4820.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4820.png,Lung_Opacity/masks/Lung_Opacity-4820.png,True,False,299,299,L,120.97,b15d759522285e80fd22fd44b956a4c2
Lung_Opacity-4821.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4821.png,Lung_Opacity/images/Lung_Opacity-4821.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4821.png,Lung_Opacity/masks/Lung_Opacity-4821.png,True,False,299,299,L,181.55,b36fc3d657de0b20aa6816328c872534
Lung_Opacity-4822.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4822.png,Lung_Opacity/images/Lung_Opacity-4822.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4822.png,Lung_Opacity/masks/Lung_Opacity-4822.png,True,False,299,299,L,153.8,1e73236b1db04d4b5936765dad97cb75
Lung_Opacity-4823.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4823.png,Lung_Opacity/images/Lung_Opacity-4823.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4823.png,Lung_Opacity/masks/Lung_Opacity-4823.png,True,False,299,299,L,160.21,2c7e6fa5aec454b2756fcd8ad0608436
Lung_Opacity-4824.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4824.png,Lung_Opacity/images/Lung_Opacity-4824.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4824.png,Lung_Opacity/masks/Lung_Opacity-4824.png,True,False,299,299,L,117.41,4cb722a206745355e0417c6ffaceceb6
Lung_Opacity-4825.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4825.png,Lung_Opacity/images/Lung_Opacity-4825.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4825.png,Lung_Opacity/masks/Lung_Opacity-4825.png,True,False,299,299,L,113.13,fc1424f97582f88d5af98f68ba6d430d
Lung_Opacity-4826.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4826.png,Lung_Opacity/images/Lung_Opacity-4826.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4826.png,Lung_Opacity/masks/Lung_Opacity-4826.png,True,False,299,299,L,108.11,ee816c4ba9204152ce808983264b242d
Lung_Opacity-4827.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4827.png,Lung_Opacity/images/Lung_Opacity-4827.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4827.png,Lung_Opacity/masks/Lung_Opacity-4827.png,True,False,299,299,L,172.71,d26fb6401d202c965b732af9e2e7b238
Lung_Opacity-4828.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4828.png,Lung_Opacity/images/Lung_Opacity-4828.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4828.png,Lung_Opacity/masks/Lung_Opacity-4828.png,True,False,299,299,L,106.95,e61d04c8d22f50e6066ccec63e818aab
Lung_Opacity-4829.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4829.png,Lung_Opacity/images/Lung_Opacity-4829.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4829.png,Lung_Opacity/masks/Lung_Opacity-4829.png,True,False,299,299,L,125.96,8bb2a146bb542467746283fb1068f8f8
Lung_Opacity-483.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-483.png,Lung_Opacity/images/Lung_Opacity-483.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-483.png,Lung_Opacity/masks/Lung_Opacity-483.png,True,False,299,299,L,158.5,53ac6add6b6c5a3b2f0e109dd9f6f5f0
Lung_Opacity-4830.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4830.png,Lung_Opacity/images/Lung_Opacity-4830.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4830.png,Lung_Opacity/masks/Lung_Opacity-4830.png,True,False,299,299,L,115.27,56b8d2654d1c3c1525132c8efaa8a20a
Lung_Opacity-4831.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4831.png,Lung_Opacity/images/Lung_Opacity-4831.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4831.png,Lung_Opacity/masks/Lung_Opacity-4831.png,True,False,299,299,L,152.16,8b89c400020f21f5de7650d522ffdd0f
Lung_Opacity-4832.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4832.png,Lung_Opacity/images/Lung_Opacity-4832.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4832.png,Lung_Opacity/masks/Lung_Opacity-4832.png,True,False,299,299,L,119.83,559641326e9f20f590f33069e98af799
Lung_Opacity-4833.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4833.png,Lung_Opacity/images/Lung_Opacity-4833.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4833.png,Lung_Opacity/masks/Lung_Opacity-4833.png,True,False,299,299,L,167.3,addaf5cf9990eb999c865d050a503fdf
Lung_Opacity-4834.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4834.png,Lung_Opacity/images/Lung_Opacity-4834.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4834.png,Lung_Opacity/masks/Lung_Opacity-4834.png,True,False,299,299,L,128.8,9c2ada28ce2cf3dd7c238283f3fc0c6a
Lung_Opacity-4835.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4835.png,Lung_Opacity/images/Lung_Opacity-4835.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4835.png,Lung_Opacity/masks/Lung_Opacity-4835.png,True,False,299,299,L,171.76,39ec350cbc4ac0436ed5f3b70f0cf30e
Lung_Opacity-4836.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4836.png,Lung_Opacity/images/Lung_Opacity-4836.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4836.png,Lung_Opacity/masks/Lung_Opacity-4836.png,True,False,299,299,L,106.39,e509bbf65144a6753ab2ae89a0731ed2
Lung_Opacity-4837.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4837.png,Lung_Opacity/images/Lung_Opacity-4837.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4837.png,Lung_Opacity/masks/Lung_Opacity-4837.png,True,False,299,299,L,166.16,a39c68c5609344933a4226af7999e7cc
Lung_Opacity-4838.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4838.png,Lung_Opacity/images/Lung_Opacity-4838.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4838.png,Lung_Opacity/masks/Lung_Opacity-4838.png,True,False,299,299,L,109.98,ca99dc4dbbb1d5c0caae627420f64e55
Lung_Opacity-4839.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4839.png,Lung_Opacity/images/Lung_Opacity-4839.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4839.png,Lung_Opacity/masks/Lung_Opacity-4839.png,True,False,299,299,L,151.1,c499e23814028a87c208287148b72450
Lung_Opacity-484.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-484.png,Lung_Opacity/images/Lung_Opacity-484.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-484.png,Lung_Opacity/masks/Lung_Opacity-484.png,True,False,299,299,L,118.91,8d00268d32f36aa33a3fc68f2b03faea
Lung_Opacity-4840.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4840.png,Lung_Opacity/images/Lung_Opacity-4840.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4840.png,Lung_Opacity/masks/Lung_Opacity-4840.png,True,False,299,299,L,117.84,886958514556ffd187563e7b4597b60e
Lung_Opacity-4841.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4841.png,Lung_Opacity/images/Lung_Opacity-4841.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4841.png,Lung_Opacity/masks/Lung_Opacity-4841.png,True,False,299,299,L,114.74,bb30040e82b30c120b6c32951c78d9b4
Lung_Opacity-4842.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4842.png,Lung_Opacity/images/Lung_Opacity-4842.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4842.png,Lung_Opacity/masks/Lung_Opacity-4842.png,True,False,299,299,L,115.61,be61be99e9c1fed5331b75939cc5ee76
Lung_Opacity-4843.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4843.png,Lung_Opacity/images/Lung_Opacity-4843.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4843.png,Lung_Opacity/masks/Lung_Opacity-4843.png,True,False,299,299,L,96.68,d55be8cddb19fee91b33d0135fd35e79
Lung_Opacity-4844.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4844.png,Lung_Opacity/images/Lung_Opacity-4844.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4844.png,Lung_Opacity/masks/Lung_Opacity-4844.png,True,False,299,299,L,103.45,8da62c40e195c83975c2c8d05cc88289
Lung_Opacity-4845.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4845.png,Lung_Opacity/images/Lung_Opacity-4845.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4845.png,Lung_Opacity/masks/Lung_Opacity-4845.png,True,False,299,299,L,133.31,a85c9673c2a1794d4d69d4c1137d270c
Lung_Opacity-4846.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4846.png,Lung_Opacity/images/Lung_Opacity-4846.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4846.png,Lung_Opacity/masks/Lung_Opacity-4846.png,True,False,299,299,L,133.47,91b0d345a196c7200643e6d233975262
Lung_Opacity-4847.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4847.png,Lung_Opacity/images/Lung_Opacity-4847.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4847.png,Lung_Opacity/masks/Lung_Opacity-4847.png,True,False,299,299,L,157.16,dafefc0c6aaf0ab3f52616272e3a5e6f
Lung_Opacity-4848.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4848.png,Lung_Opacity/images/Lung_Opacity-4848.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4848.png,Lung_Opacity/masks/Lung_Opacity-4848.png,True,False,299,299,L,108.74,37a8b739170879b10905150f25146dfb
Lung_Opacity-4849.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4849.png,Lung_Opacity/images/Lung_Opacity-4849.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4849.png,Lung_Opacity/masks/Lung_Opacity-4849.png,True,False,299,299,L,169.63,b12a6e2751294a2c4d8a550213f2e294
Lung_Opacity-485.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-485.png,Lung_Opacity/images/Lung_Opacity-485.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-485.png,Lung_Opacity/masks/Lung_Opacity-485.png,True,False,299,299,L,110.08,8db30e25a178e722b4ac1fdda02b273c
Lung_Opacity-4850.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4850.png,Lung_Opacity/images/Lung_Opacity-4850.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4850.png,Lung_Opacity/masks/Lung_Opacity-4850.png,True,False,299,299,L,108.96,9e6f4ed2864ce1e13d0b742fdc6ffaf8
Lung_Opacity-4851.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4851.png,Lung_Opacity/images/Lung_Opacity-4851.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4851.png,Lung_Opacity/masks/Lung_Opacity-4851.png,True,False,299,299,L,92.53,b703f9113c627ce9d233a51d9fac00a0
Lung_Opacity-4852.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4852.png,Lung_Opacity/images/Lung_Opacity-4852.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4852.png,Lung_Opacity/masks/Lung_Opacity-4852.png,True,False,299,299,L,113.22,fd63c4888df535b2cfd6e41fb09b0b82
Lung_Opacity-4853.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4853.png,Lung_Opacity/images/Lung_Opacity-4853.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4853.png,Lung_Opacity/masks/Lung_Opacity-4853.png,True,False,299,299,L,121.35,290b6e702c9e2a959a52d46d0f9dd0cf
Lung_Opacity-4854.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4854.png,Lung_Opacity/images/Lung_Opacity-4854.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4854.png,Lung_Opacity/masks/Lung_Opacity-4854.png,True,False,299,299,L,165.06,10177672c838a0d77811553cdf9be2e4
Lung_Opacity-4855.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4855.png,Lung_Opacity/images/Lung_Opacity-4855.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4855.png,Lung_Opacity/masks/Lung_Opacity-4855.png,True,False,299,299,L,115.06,dfd1bb712b395a288fad94c9159f7565
Lung_Opacity-4856.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4856.png,Lung_Opacity/images/Lung_Opacity-4856.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4856.png,Lung_Opacity/masks/Lung_Opacity-4856.png,True,False,299,299,L,118.38,c598a78ad0ce85c275704ec2897f92ad
Lung_Opacity-4857.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4857.png,Lung_Opacity/images/Lung_Opacity-4857.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4857.png,Lung_Opacity/masks/Lung_Opacity-4857.png,True,False,299,299,L,145.37,2c58174f404a7c6747bb66a71ab37843
Lung_Opacity-4858.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4858.png,Lung_Opacity/images/Lung_Opacity-4858.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4858.png,Lung_Opacity/masks/Lung_Opacity-4858.png,True,False,299,299,L,119.52,d3d3087d287302b4fcc40e7d3981f831
Lung_Opacity-4859.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4859.png,Lung_Opacity/images/Lung_Opacity-4859.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4859.png,Lung_Opacity/masks/Lung_Opacity-4859.png,True,False,299,299,L,159.2,9a054197027d0b90288e2e1cf7c252d2
Lung_Opacity-486.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-486.png,Lung_Opacity/images/Lung_Opacity-486.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-486.png,Lung_Opacity/masks/Lung_Opacity-486.png,True,False,299,299,L,116.01,79ccd61e749e1b73fef8cdf27121f863
Lung_Opacity-4860.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4860.png,Lung_Opacity/images/Lung_Opacity-4860.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4860.png,Lung_Opacity/masks/Lung_Opacity-4860.png,True,False,299,299,L,121.15,9c011b52bc3b80b8097533f1126b681d
Lung_Opacity-4861.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4861.png,Lung_Opacity/images/Lung_Opacity-4861.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4861.png,Lung_Opacity/masks/Lung_Opacity-4861.png,True,False,299,299,L,82.89,a4cac88e8008e8a26bb7db9efb35a897
Lung_Opacity-4862.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4862.png,Lung_Opacity/images/Lung_Opacity-4862.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4862.png,Lung_Opacity/masks/Lung_Opacity-4862.png,True,False,299,299,L,103.54,28b257f49ebea51b3b0abf83fc994577
Lung_Opacity-4863.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4863.png,Lung_Opacity/images/Lung_Opacity-4863.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4863.png,Lung_Opacity/masks/Lung_Opacity-4863.png,True,False,299,299,L,98.84,81356da43088b3479f63ac231590184b
Lung_Opacity-4864.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4864.png,Lung_Opacity/images/Lung_Opacity-4864.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4864.png,Lung_Opacity/masks/Lung_Opacity-4864.png,True,False,299,299,L,125.04,bbd5402017e8df530fdf5bf8eb666e19
Lung_Opacity-4865.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4865.png,Lung_Opacity/images/Lung_Opacity-4865.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4865.png,Lung_Opacity/masks/Lung_Opacity-4865.png,True,False,299,299,L,162.01,18fd6936d265136337f698d14a2d45b1
Lung_Opacity-4866.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4866.png,Lung_Opacity/images/Lung_Opacity-4866.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4866.png,Lung_Opacity/masks/Lung_Opacity-4866.png,True,False,299,299,L,111.91,e911d293ad9e40c638f69560eba703f9
Lung_Opacity-4867.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4867.png,Lung_Opacity/images/Lung_Opacity-4867.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4867.png,Lung_Opacity/masks/Lung_Opacity-4867.png,True,False,299,299,L,108.1,1204585fcd65572eed659df76b79bbe8
Lung_Opacity-4868.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4868.png,Lung_Opacity/images/Lung_Opacity-4868.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4868.png,Lung_Opacity/masks/Lung_Opacity-4868.png,True,False,299,299,L,151.36,4820e67a7b9c842c71919f1c28de5573
Lung_Opacity-4869.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4869.png,Lung_Opacity/images/Lung_Opacity-4869.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4869.png,Lung_Opacity/masks/Lung_Opacity-4869.png,True,False,299,299,L,128.05,4b84ca617037e059ab3b0926ed565b17
Lung_Opacity-487.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-487.png,Lung_Opacity/images/Lung_Opacity-487.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-487.png,Lung_Opacity/masks/Lung_Opacity-487.png,True,False,299,299,L,115.13,81e8f13f99dc95d33ec89adaf9f0c408
Lung_Opacity-4870.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4870.png,Lung_Opacity/images/Lung_Opacity-4870.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4870.png,Lung_Opacity/masks/Lung_Opacity-4870.png,True,False,299,299,L,117.73,fc4fe336fa4a01ba74aea30a6ee2d72f
Lung_Opacity-4871.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4871.png,Lung_Opacity/images/Lung_Opacity-4871.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4871.png,Lung_Opacity/masks/Lung_Opacity-4871.png,True,False,299,299,L,126.69,6fe692d6527608d52d406c442f4047c3
Lung_Opacity-4872.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4872.png,Lung_Opacity/images/Lung_Opacity-4872.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4872.png,Lung_Opacity/masks/Lung_Opacity-4872.png,True,False,299,299,L,133.27,a9534c43b8d5f2693ad228bfe2dfd5fc
Lung_Opacity-4873.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4873.png,Lung_Opacity/images/Lung_Opacity-4873.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4873.png,Lung_Opacity/masks/Lung_Opacity-4873.png,True,False,299,299,L,167.26,4116fe942c87edd684d13fda50206801
Lung_Opacity-4874.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4874.png,Lung_Opacity/images/Lung_Opacity-4874.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4874.png,Lung_Opacity/masks/Lung_Opacity-4874.png,True,False,299,299,L,163.17,55148e8765f2b86d30d94a63c9a5aca5
Lung_Opacity-4875.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4875.png,Lung_Opacity/images/Lung_Opacity-4875.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4875.png,Lung_Opacity/masks/Lung_Opacity-4875.png,True,False,299,299,L,111.5,107f6d595a5ce504189ca997ccfa1fdb
Lung_Opacity-4876.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4876.png,Lung_Opacity/images/Lung_Opacity-4876.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4876.png,Lung_Opacity/masks/Lung_Opacity-4876.png,True,False,299,299,L,164.2,c43ca9e50ed8925b7f988dd88b112d80
Lung_Opacity-4877.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4877.png,Lung_Opacity/images/Lung_Opacity-4877.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4877.png,Lung_Opacity/masks/Lung_Opacity-4877.png,True,False,299,299,L,108.29,ec270d631848339da03e00b0f02aad42
Lung_Opacity-4878.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4878.png,Lung_Opacity/images/Lung_Opacity-4878.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4878.png,Lung_Opacity/masks/Lung_Opacity-4878.png,True,False,299,299,L,171.69,03f63859387750960d64878565ba9e3f
Lung_Opacity-4879.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4879.png,Lung_Opacity/images/Lung_Opacity-4879.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4879.png,Lung_Opacity/masks/Lung_Opacity-4879.png,True,False,299,299,L,107.72,f15df3a1efff1940fb7aa8a3980a7800
Lung_Opacity-488.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-488.png,Lung_Opacity/images/Lung_Opacity-488.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-488.png,Lung_Opacity/masks/Lung_Opacity-488.png,True,False,299,299,L,115.84,52264876df0fc72e6714c2b0d4a96f28
Lung_Opacity-4880.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4880.png,Lung_Opacity/images/Lung_Opacity-4880.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4880.png,Lung_Opacity/masks/Lung_Opacity-4880.png,True,False,299,299,L,134.91,8bdea5e208308f6bfe4eb2d636dc5186
Lung_Opacity-4881.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4881.png,Lung_Opacity/images/Lung_Opacity-4881.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4881.png,Lung_Opacity/masks/Lung_Opacity-4881.png,True,False,299,299,L,108.44,321596bed87d394775b87ac2174e5f4b
Lung_Opacity-4882.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4882.png,Lung_Opacity/images/Lung_Opacity-4882.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4882.png,Lung_Opacity/masks/Lung_Opacity-4882.png,True,False,299,299,L,117.35,4e96ad688c0f618a009c0677088d07b4
Lung_Opacity-4883.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4883.png,Lung_Opacity/images/Lung_Opacity-4883.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4883.png,Lung_Opacity/masks/Lung_Opacity-4883.png,True,False,299,299,L,102.78,28ec0e759b1f4d803b6d475fb056367d
Lung_Opacity-4884.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4884.png,Lung_Opacity/images/Lung_Opacity-4884.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4884.png,Lung_Opacity/masks/Lung_Opacity-4884.png,True,False,299,299,L,121.71,cdc5c097076cf401a93feba39a0368ce
Lung_Opacity-4885.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4885.png,Lung_Opacity/images/Lung_Opacity-4885.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4885.png,Lung_Opacity/masks/Lung_Opacity-4885.png,True,False,299,299,L,148.25,1c80b8b0b11e0a030c74a64c097523a8
Lung_Opacity-4886.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4886.png,Lung_Opacity/images/Lung_Opacity-4886.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4886.png,Lung_Opacity/masks/Lung_Opacity-4886.png,True,False,299,299,L,163.41,e2ad3197f819939d731659acdb41f7db
Lung_Opacity-4887.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4887.png,Lung_Opacity/images/Lung_Opacity-4887.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4887.png,Lung_Opacity/masks/Lung_Opacity-4887.png,True,False,299,299,L,184.23,6db06ccc5803aff2fa6730d7cf45c760
Lung_Opacity-4888.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4888.png,Lung_Opacity/images/Lung_Opacity-4888.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4888.png,Lung_Opacity/masks/Lung_Opacity-4888.png,True,False,299,299,L,113.47,59ac642a47fa36d00789e447ce1ca9d5
Lung_Opacity-4889.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4889.png,Lung_Opacity/images/Lung_Opacity-4889.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4889.png,Lung_Opacity/masks/Lung_Opacity-4889.png,True,False,299,299,L,115.51,fd1be07221dee876802e952150f8c457
Lung_Opacity-489.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-489.png,Lung_Opacity/images/Lung_Opacity-489.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-489.png,Lung_Opacity/masks/Lung_Opacity-489.png,True,False,299,299,L,190.09,1858e9046fec60afba723dda18d37359
Lung_Opacity-4890.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4890.png,Lung_Opacity/images/Lung_Opacity-4890.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4890.png,Lung_Opacity/masks/Lung_Opacity-4890.png,True,False,299,299,L,124.64,0cd1f1e37dd66b9b9f3916b3ac2c7884
Lung_Opacity-4891.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4891.png,Lung_Opacity/images/Lung_Opacity-4891.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4891.png,Lung_Opacity/masks/Lung_Opacity-4891.png,True,False,299,299,L,158.23,5479e443a06b5a3530d313d0bde8bfe3
Lung_Opacity-4892.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4892.png,Lung_Opacity/images/Lung_Opacity-4892.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4892.png,Lung_Opacity/masks/Lung_Opacity-4892.png,True,False,299,299,L,117.26,6904ab6a6ef89a4818f937e0908ace5b
Lung_Opacity-4893.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4893.png,Lung_Opacity/images/Lung_Opacity-4893.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4893.png,Lung_Opacity/masks/Lung_Opacity-4893.png,True,False,299,299,L,121.77,4265648fd96762fa86229968442f9111
Lung_Opacity-4894.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4894.png,Lung_Opacity/images/Lung_Opacity-4894.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4894.png,Lung_Opacity/masks/Lung_Opacity-4894.png,True,False,299,299,L,147.36,664ab069406f1a7be9be5dfb6c457a80
Lung_Opacity-4895.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4895.png,Lung_Opacity/images/Lung_Opacity-4895.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4895.png,Lung_Opacity/masks/Lung_Opacity-4895.png,True,False,299,299,L,160.84,08b044f32aabff6a991fdb9cf763044c
Lung_Opacity-4896.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4896.png,Lung_Opacity/images/Lung_Opacity-4896.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4896.png,Lung_Opacity/masks/Lung_Opacity-4896.png,True,False,299,299,L,160.22,c3e6c1a7211218b7c539c16eb4b5dd15
Lung_Opacity-4897.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4897.png,Lung_Opacity/images/Lung_Opacity-4897.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4897.png,Lung_Opacity/masks/Lung_Opacity-4897.png,True,False,299,299,L,171.76,a53aab2248a7917efc858613c2d84bd1
Lung_Opacity-4898.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4898.png,Lung_Opacity/images/Lung_Opacity-4898.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4898.png,Lung_Opacity/masks/Lung_Opacity-4898.png,True,False,299,299,L,172.59,239ff40e2a4a2b613c373f76b094a044
Lung_Opacity-4899.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4899.png,Lung_Opacity/images/Lung_Opacity-4899.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4899.png,Lung_Opacity/masks/Lung_Opacity-4899.png,True,False,299,299,L,169.78,266c0b2871ef1d64b2d33c46db49d421
Lung_Opacity-49.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-49.png,Lung_Opacity/images/Lung_Opacity-49.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-49.png,Lung_Opacity/masks/Lung_Opacity-49.png,True,False,299,299,L,119.12,352b62c51059d34440a4628e6000cce5
Lung_Opacity-490.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-490.png,Lung_Opacity/images/Lung_Opacity-490.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-490.png,Lung_Opacity/masks/Lung_Opacity-490.png,True,False,299,299,L,124.08,49c73a3308c18996805acb726f1a779e
Lung_Opacity-4900.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4900.png,Lung_Opacity/images/Lung_Opacity-4900.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4900.png,Lung_Opacity/masks/Lung_Opacity-4900.png,True,False,299,299,L,109.96,bbd2a837e390332f44e70b25d53fe679
Lung_Opacity-4901.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4901.png,Lung_Opacity/images/Lung_Opacity-4901.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4901.png,Lung_Opacity/masks/Lung_Opacity-4901.png,True,False,299,299,L,130.76,b907d39a93a788241d9b7979d0042957
Lung_Opacity-4902.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4902.png,Lung_Opacity/images/Lung_Opacity-4902.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4902.png,Lung_Opacity/masks/Lung_Opacity-4902.png,True,False,299,299,L,138.6,2f824f2108ea761e94ba7ba12b9c2e3c
Lung_Opacity-4903.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4903.png,Lung_Opacity/images/Lung_Opacity-4903.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4903.png,Lung_Opacity/masks/Lung_Opacity-4903.png,True,False,299,299,L,151.05,604a9e0747cb405189286b587bd46063
Lung_Opacity-4904.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4904.png,Lung_Opacity/images/Lung_Opacity-4904.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4904.png,Lung_Opacity/masks/Lung_Opacity-4904.png,True,False,299,299,L,105.83,965a2c6fc35a57027925a0f00524fbfd
Lung_Opacity-4905.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4905.png,Lung_Opacity/images/Lung_Opacity-4905.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4905.png,Lung_Opacity/masks/Lung_Opacity-4905.png,True,False,299,299,L,104.42,fae9c896fce4e90c3eb6bd32008af3c8
Lung_Opacity-4906.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4906.png,Lung_Opacity/images/Lung_Opacity-4906.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4906.png,Lung_Opacity/masks/Lung_Opacity-4906.png,True,False,299,299,L,165.15,c1675ae835e98099b18dfd057ea22cb6
Lung_Opacity-4907.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4907.png,Lung_Opacity/images/Lung_Opacity-4907.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4907.png,Lung_Opacity/masks/Lung_Opacity-4907.png,True,False,299,299,L,117.84,546eb81e664a0f1012b659f75f461879
Lung_Opacity-4908.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4908.png,Lung_Opacity/images/Lung_Opacity-4908.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4908.png,Lung_Opacity/masks/Lung_Opacity-4908.png,True,False,299,299,L,110.29,ea3619cc442164fb98f5fdf25b4093ba
Lung_Opacity-4909.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4909.png,Lung_Opacity/images/Lung_Opacity-4909.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4909.png,Lung_Opacity/masks/Lung_Opacity-4909.png,True,False,299,299,L,112.76,9bfad4ac5863d18e7e71113a3698e769
Lung_Opacity-491.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-491.png,Lung_Opacity/images/Lung_Opacity-491.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-491.png,Lung_Opacity/masks/Lung_Opacity-491.png,True,False,299,299,L,94.66,27f130ee9a29605b13eb684da4ccb60d
Lung_Opacity-4910.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4910.png,Lung_Opacity/images/Lung_Opacity-4910.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4910.png,Lung_Opacity/masks/Lung_Opacity-4910.png,True,False,299,299,L,107.63,07cb88cfcdcc1e55eae58569a401a69d
Lung_Opacity-4911.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4911.png,Lung_Opacity/images/Lung_Opacity-4911.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4911.png,Lung_Opacity/masks/Lung_Opacity-4911.png,True,False,299,299,L,108.24,035cc70264399206f177cad980641c36
Lung_Opacity-4912.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4912.png,Lung_Opacity/images/Lung_Opacity-4912.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4912.png,Lung_Opacity/masks/Lung_Opacity-4912.png,True,False,299,299,L,137.87,14f2146dbcb51c0cf23495bb5786e0fa
Lung_Opacity-4913.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4913.png,Lung_Opacity/images/Lung_Opacity-4913.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4913.png,Lung_Opacity/masks/Lung_Opacity-4913.png,True,False,299,299,L,134.44,c758d0b681555bc55738dbf96da72ef1
Lung_Opacity-4914.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4914.png,Lung_Opacity/images/Lung_Opacity-4914.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4914.png,Lung_Opacity/masks/Lung_Opacity-4914.png,True,False,299,299,L,145.65,2efafc570acdaba9fef37555202b37fe
Lung_Opacity-4915.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4915.png,Lung_Opacity/images/Lung_Opacity-4915.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4915.png,Lung_Opacity/masks/Lung_Opacity-4915.png,True,False,299,299,L,92.69,a5e2c7df9118b9d4be227c60d75c19db
Lung_Opacity-4916.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4916.png,Lung_Opacity/images/Lung_Opacity-4916.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4916.png,Lung_Opacity/masks/Lung_Opacity-4916.png,True,False,299,299,L,122.07,2ceac37d4e819bdde4b721f8404f5007
Lung_Opacity-4917.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4917.png,Lung_Opacity/images/Lung_Opacity-4917.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4917.png,Lung_Opacity/masks/Lung_Opacity-4917.png,True,False,299,299,L,115.13,ecc9dd42a73d2594c780cf05aa376916
Lung_Opacity-4918.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4918.png,Lung_Opacity/images/Lung_Opacity-4918.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4918.png,Lung_Opacity/masks/Lung_Opacity-4918.png,True,False,299,299,L,157.24,3a4f25a0e3ffd0fae1d885699a4136b3
Lung_Opacity-4919.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4919.png,Lung_Opacity/images/Lung_Opacity-4919.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4919.png,Lung_Opacity/masks/Lung_Opacity-4919.png,True,False,299,299,L,154.91,eaf4b1061002d5522ea09295d10d245e
Lung_Opacity-492.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-492.png,Lung_Opacity/images/Lung_Opacity-492.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-492.png,Lung_Opacity/masks/Lung_Opacity-492.png,True,False,299,299,L,142.88,1da4f4d027327b120903a23c1c7edbad
Lung_Opacity-4920.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4920.png,Lung_Opacity/images/Lung_Opacity-4920.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4920.png,Lung_Opacity/masks/Lung_Opacity-4920.png,True,False,299,299,L,105.22,403e1b9c0cb22b836e295d6b2a3b8ffb
Lung_Opacity-4921.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4921.png,Lung_Opacity/images/Lung_Opacity-4921.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4921.png,Lung_Opacity/masks/Lung_Opacity-4921.png,True,False,299,299,L,173.39,b06f7e01e428560e5ee1ac79c179f9f2
Lung_Opacity-4922.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4922.png,Lung_Opacity/images/Lung_Opacity-4922.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4922.png,Lung_Opacity/masks/Lung_Opacity-4922.png,True,False,299,299,L,113.97,06e7c489a3a71d672cf86f92a4304308
Lung_Opacity-4923.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4923.png,Lung_Opacity/images/Lung_Opacity-4923.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4923.png,Lung_Opacity/masks/Lung_Opacity-4923.png,True,False,299,299,L,122.0,7d1523063ea9bc22488a8795222b2e72
Lung_Opacity-4924.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4924.png,Lung_Opacity/images/Lung_Opacity-4924.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4924.png,Lung_Opacity/masks/Lung_Opacity-4924.png,True,False,299,299,L,111.07,b261c4c35200d6fd4873840a7743a941
Lung_Opacity-4925.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4925.png,Lung_Opacity/images/Lung_Opacity-4925.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4925.png,Lung_Opacity/masks/Lung_Opacity-4925.png,True,False,299,299,L,146.36,c6c26627cc417a91efcc52a0574e25e8
Lung_Opacity-4926.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4926.png,Lung_Opacity/images/Lung_Opacity-4926.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4926.png,Lung_Opacity/masks/Lung_Opacity-4926.png,True,False,299,299,L,120.11,71e03f3e484324fb6da1a3f6cb1b7df0
Lung_Opacity-4927.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4927.png,Lung_Opacity/images/Lung_Opacity-4927.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4927.png,Lung_Opacity/masks/Lung_Opacity-4927.png,True,False,299,299,L,148.21,25873164180d472f7aa23014dc0a6929
Lung_Opacity-4928.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4928.png,Lung_Opacity/images/Lung_Opacity-4928.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4928.png,Lung_Opacity/masks/Lung_Opacity-4928.png,True,False,299,299,L,89.46,d363edcd9c6ebefd360a155a842036d7
Lung_Opacity-4929.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4929.png,Lung_Opacity/images/Lung_Opacity-4929.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4929.png,Lung_Opacity/masks/Lung_Opacity-4929.png,True,False,299,299,L,169.13,efc16fdbae3984ccfb8c3b1a8fed8d55
Lung_Opacity-493.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-493.png,Lung_Opacity/images/Lung_Opacity-493.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-493.png,Lung_Opacity/masks/Lung_Opacity-493.png,True,False,299,299,L,82.55,bc46f249d1b71264f36be8a4fa4c17a0
Lung_Opacity-4930.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4930.png,Lung_Opacity/images/Lung_Opacity-4930.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4930.png,Lung_Opacity/masks/Lung_Opacity-4930.png,True,False,299,299,L,116.63,aa7b57078e1b38737a6c645d28e272ae
Lung_Opacity-4931.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4931.png,Lung_Opacity/images/Lung_Opacity-4931.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4931.png,Lung_Opacity/masks/Lung_Opacity-4931.png,True,False,299,299,L,96.52,1644d89daa1ec3058bc623d9d6580a8b
Lung_Opacity-4932.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4932.png,Lung_Opacity/images/Lung_Opacity-4932.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4932.png,Lung_Opacity/masks/Lung_Opacity-4932.png,True,False,299,299,L,115.76,ba8f39e7bf9b3ce1365335be5a5d60be
Lung_Opacity-4933.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4933.png,Lung_Opacity/images/Lung_Opacity-4933.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4933.png,Lung_Opacity/masks/Lung_Opacity-4933.png,True,False,299,299,L,113.95,85312646ef6b2f52a3747784f1722145
Lung_Opacity-4934.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4934.png,Lung_Opacity/images/Lung_Opacity-4934.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4934.png,Lung_Opacity/masks/Lung_Opacity-4934.png,True,False,299,299,L,86.21,1034b3cbeca87010a6e60c309e944a02
Lung_Opacity-4935.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4935.png,Lung_Opacity/images/Lung_Opacity-4935.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4935.png,Lung_Opacity/masks/Lung_Opacity-4935.png,True,False,299,299,L,173.75,2f94e9256544c41dc7ab52813975c3e5
Lung_Opacity-4936.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4936.png,Lung_Opacity/images/Lung_Opacity-4936.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4936.png,Lung_Opacity/masks/Lung_Opacity-4936.png,True,False,299,299,L,160.4,e04961e0697a0f0318652a54d130c372
Lung_Opacity-4937.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4937.png,Lung_Opacity/images/Lung_Opacity-4937.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4937.png,Lung_Opacity/masks/Lung_Opacity-4937.png,True,False,299,299,L,129.28,facd383e518a83c6bf0be5ced580ff84
Lung_Opacity-4938.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4938.png,Lung_Opacity/images/Lung_Opacity-4938.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4938.png,Lung_Opacity/masks/Lung_Opacity-4938.png,True,False,299,299,L,176.14,0d570fe652f751639c4144d623ef672e
Lung_Opacity-4939.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4939.png,Lung_Opacity/images/Lung_Opacity-4939.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4939.png,Lung_Opacity/masks/Lung_Opacity-4939.png,True,False,299,299,L,152.85,be53322ac0d1558184844a8650ae9a52
Lung_Opacity-494.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-494.png,Lung_Opacity/images/Lung_Opacity-494.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-494.png,Lung_Opacity/masks/Lung_Opacity-494.png,True,False,299,299,L,121.05,4d0720c8286c95c7336125b1afdef181
Lung_Opacity-4940.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4940.png,Lung_Opacity/images/Lung_Opacity-4940.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4940.png,Lung_Opacity/masks/Lung_Opacity-4940.png,True,False,299,299,L,113.53,cd532d26debe2d0299f4c7e128f8a3a5
Lung_Opacity-4941.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4941.png,Lung_Opacity/images/Lung_Opacity-4941.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4941.png,Lung_Opacity/masks/Lung_Opacity-4941.png,True,False,299,299,L,171.0,2d246081507cbbb8ed8fcc049b3643ff
Lung_Opacity-4942.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4942.png,Lung_Opacity/images/Lung_Opacity-4942.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4942.png,Lung_Opacity/masks/Lung_Opacity-4942.png,True,False,299,299,L,147.21,5a3e66b3f0c2f5337a5ca8f7a521d7a4
Lung_Opacity-4943.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4943.png,Lung_Opacity/images/Lung_Opacity-4943.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4943.png,Lung_Opacity/masks/Lung_Opacity-4943.png,True,False,299,299,L,116.38,cfcd39aa2e01f75d231f0f93864d0902
Lung_Opacity-4944.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4944.png,Lung_Opacity/images/Lung_Opacity-4944.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4944.png,Lung_Opacity/masks/Lung_Opacity-4944.png,True,False,299,299,L,161.0,5cab09dfc2b43780cd65a5fea9884bc7
Lung_Opacity-4945.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4945.png,Lung_Opacity/images/Lung_Opacity-4945.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4945.png,Lung_Opacity/masks/Lung_Opacity-4945.png,True,False,299,299,L,111.31,d205f767c8c931dc766f91b21143aa06
Lung_Opacity-4946.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4946.png,Lung_Opacity/images/Lung_Opacity-4946.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4946.png,Lung_Opacity/masks/Lung_Opacity-4946.png,True,False,299,299,L,113.78,9e182182a462d6adfcc3d0ab1aff4060
Lung_Opacity-4947.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4947.png,Lung_Opacity/images/Lung_Opacity-4947.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4947.png,Lung_Opacity/masks/Lung_Opacity-4947.png,True,False,299,299,L,113.31,a3feb43866013a77198d09b89d0cba72
Lung_Opacity-4948.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4948.png,Lung_Opacity/images/Lung_Opacity-4948.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4948.png,Lung_Opacity/masks/Lung_Opacity-4948.png,True,False,299,299,L,118.9,1e9d20ef6e10930ea867c628a0ca34c3
Lung_Opacity-4949.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4949.png,Lung_Opacity/images/Lung_Opacity-4949.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4949.png,Lung_Opacity/masks/Lung_Opacity-4949.png,True,False,299,299,L,109.54,0973e451405045320012bea10330f2a2
Lung_Opacity-495.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-495.png,Lung_Opacity/images/Lung_Opacity-495.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-495.png,Lung_Opacity/masks/Lung_Opacity-495.png,True,False,299,299,L,141.68,2f5e6bce44d4b63daeaded6cf6042c22
Lung_Opacity-4950.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4950.png,Lung_Opacity/images/Lung_Opacity-4950.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4950.png,Lung_Opacity/masks/Lung_Opacity-4950.png,True,False,299,299,L,114.68,5b94f65194888b72402a5fba036adb32
Lung_Opacity-4951.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4951.png,Lung_Opacity/images/Lung_Opacity-4951.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4951.png,Lung_Opacity/masks/Lung_Opacity-4951.png,True,False,299,299,L,122.33,76f9778617bd69325fa611f6ce99bb5d
Lung_Opacity-4952.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4952.png,Lung_Opacity/images/Lung_Opacity-4952.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4952.png,Lung_Opacity/masks/Lung_Opacity-4952.png,True,False,299,299,L,125.82,207a289f45a584a03107c7d01b7bce69
Lung_Opacity-4953.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4953.png,Lung_Opacity/images/Lung_Opacity-4953.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4953.png,Lung_Opacity/masks/Lung_Opacity-4953.png,True,False,299,299,L,120.36,741af3d9c32b551d8e69a1fc905c6b75
Lung_Opacity-4954.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4954.png,Lung_Opacity/images/Lung_Opacity-4954.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4954.png,Lung_Opacity/masks/Lung_Opacity-4954.png,True,False,299,299,L,109.45,796cb7483df29870b15cd6129375904b
Lung_Opacity-4955.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4955.png,Lung_Opacity/images/Lung_Opacity-4955.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4955.png,Lung_Opacity/masks/Lung_Opacity-4955.png,True,False,299,299,L,113.06,e9b4249216b82c24fc47fdd7b2ab8356
Lung_Opacity-4956.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4956.png,Lung_Opacity/images/Lung_Opacity-4956.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4956.png,Lung_Opacity/masks/Lung_Opacity-4956.png,True,False,299,299,L,119.0,01b2dacfeb05f714e44730c03d68aafc
Lung_Opacity-4957.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4957.png,Lung_Opacity/images/Lung_Opacity-4957.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4957.png,Lung_Opacity/masks/Lung_Opacity-4957.png,True,False,299,299,L,136.33,efa11d0613564d952612464a7e945acd
Lung_Opacity-4958.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4958.png,Lung_Opacity/images/Lung_Opacity-4958.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4958.png,Lung_Opacity/masks/Lung_Opacity-4958.png,True,False,299,299,L,112.42,a2753ff57601c16045aba3db8b0d4383
Lung_Opacity-4959.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4959.png,Lung_Opacity/images/Lung_Opacity-4959.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4959.png,Lung_Opacity/masks/Lung_Opacity-4959.png,True,False,299,299,L,114.09,8bfc976e9d2cc9c9ca57a19372236137
Lung_Opacity-496.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-496.png,Lung_Opacity/images/Lung_Opacity-496.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-496.png,Lung_Opacity/masks/Lung_Opacity-496.png,True,False,299,299,L,111.94,923139e01002cb10d28912e0809e1867
Lung_Opacity-4960.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4960.png,Lung_Opacity/images/Lung_Opacity-4960.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4960.png,Lung_Opacity/masks/Lung_Opacity-4960.png,True,False,299,299,L,145.26,61022831ee18b6e108cd76a24a7bdafc
Lung_Opacity-4961.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4961.png,Lung_Opacity/images/Lung_Opacity-4961.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4961.png,Lung_Opacity/masks/Lung_Opacity-4961.png,True,False,299,299,L,88.37,27e11bb430668e853d1699f9ead6ed24
Lung_Opacity-4962.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4962.png,Lung_Opacity/images/Lung_Opacity-4962.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4962.png,Lung_Opacity/masks/Lung_Opacity-4962.png,True,False,299,299,L,172.01,a96c1e09b027d7a5241e170fcd3ec460
Lung_Opacity-4963.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4963.png,Lung_Opacity/images/Lung_Opacity-4963.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4963.png,Lung_Opacity/masks/Lung_Opacity-4963.png,True,False,299,299,L,132.84,7f2be3cea91b1853ea381a146941e5de
Lung_Opacity-4964.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4964.png,Lung_Opacity/images/Lung_Opacity-4964.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4964.png,Lung_Opacity/masks/Lung_Opacity-4964.png,True,False,299,299,L,68.41,a4079f2e0c81a3d9e23068d53208c04b
Lung_Opacity-4965.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4965.png,Lung_Opacity/images/Lung_Opacity-4965.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4965.png,Lung_Opacity/masks/Lung_Opacity-4965.png,True,False,299,299,L,156.08,062124164c59f765f9beffa652a4ddc2
Lung_Opacity-4966.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4966.png,Lung_Opacity/images/Lung_Opacity-4966.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4966.png,Lung_Opacity/masks/Lung_Opacity-4966.png,True,False,299,299,L,109.14,ac982b4ffbc03fcbb33bb9d0bfcdb958
Lung_Opacity-4967.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4967.png,Lung_Opacity/images/Lung_Opacity-4967.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4967.png,Lung_Opacity/masks/Lung_Opacity-4967.png,True,False,299,299,L,144.62,c7e14c5d351f9e1c5564c539358134e1
Lung_Opacity-4968.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4968.png,Lung_Opacity/images/Lung_Opacity-4968.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4968.png,Lung_Opacity/masks/Lung_Opacity-4968.png,True,False,299,299,L,125.97,3f1fff88ebb1c8a273297bbd888414ef
Lung_Opacity-4969.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4969.png,Lung_Opacity/images/Lung_Opacity-4969.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4969.png,Lung_Opacity/masks/Lung_Opacity-4969.png,True,False,299,299,L,64.1,791d4d594ca5794c23f9c159937969c1
Lung_Opacity-497.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-497.png,Lung_Opacity/images/Lung_Opacity-497.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-497.png,Lung_Opacity/masks/Lung_Opacity-497.png,True,False,299,299,L,137.34,d539319f3daf66ff6fa321716cf0e8f4
Lung_Opacity-4970.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4970.png,Lung_Opacity/images/Lung_Opacity-4970.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4970.png,Lung_Opacity/masks/Lung_Opacity-4970.png,True,False,299,299,L,141.5,fccc0786ee9deec6758001a8612d5184
Lung_Opacity-4971.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4971.png,Lung_Opacity/images/Lung_Opacity-4971.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4971.png,Lung_Opacity/masks/Lung_Opacity-4971.png,True,False,299,299,L,137.25,1f3adaf8987a9fa70df6bc5f0d15fd84
Lung_Opacity-4972.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4972.png,Lung_Opacity/images/Lung_Opacity-4972.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4972.png,Lung_Opacity/masks/Lung_Opacity-4972.png,True,False,299,299,L,107.97,55e684f89f55f0d10f9572b4bc3fcce8
Lung_Opacity-4973.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4973.png,Lung_Opacity/images/Lung_Opacity-4973.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4973.png,Lung_Opacity/masks/Lung_Opacity-4973.png,True,False,299,299,L,99.83,b51bd59dfef0941df8dc31c569b0ec1a
Lung_Opacity-4974.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4974.png,Lung_Opacity/images/Lung_Opacity-4974.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4974.png,Lung_Opacity/masks/Lung_Opacity-4974.png,True,False,299,299,L,163.45,74d6349f8a548cdd486a60e2f33dd5ce
Lung_Opacity-4975.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4975.png,Lung_Opacity/images/Lung_Opacity-4975.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4975.png,Lung_Opacity/masks/Lung_Opacity-4975.png,True,False,299,299,L,78.32,70fa763b64db7544b81535c0a6fe90f4
Lung_Opacity-4976.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4976.png,Lung_Opacity/images/Lung_Opacity-4976.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4976.png,Lung_Opacity/masks/Lung_Opacity-4976.png,True,False,299,299,L,105.54,946d06eb4f31f45e4994f913804bbb16
Lung_Opacity-4977.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4977.png,Lung_Opacity/images/Lung_Opacity-4977.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4977.png,Lung_Opacity/masks/Lung_Opacity-4977.png,True,False,299,299,L,107.63,0370d471d6ebba7453737fd9e7210a2b
Lung_Opacity-4978.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4978.png,Lung_Opacity/images/Lung_Opacity-4978.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4978.png,Lung_Opacity/masks/Lung_Opacity-4978.png,True,False,299,299,L,126.15,94d8069eaae58e122f61941c5b0a5a5c
Lung_Opacity-4979.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4979.png,Lung_Opacity/images/Lung_Opacity-4979.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4979.png,Lung_Opacity/masks/Lung_Opacity-4979.png,True,False,299,299,L,119.55,7dfec129b71346f568821bdf8568db2a
Lung_Opacity-498.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-498.png,Lung_Opacity/images/Lung_Opacity-498.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-498.png,Lung_Opacity/masks/Lung_Opacity-498.png,True,False,299,299,L,102.44,c75fb995dbcf560e97fc3ba2b4bb858c
Lung_Opacity-4980.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4980.png,Lung_Opacity/images/Lung_Opacity-4980.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4980.png,Lung_Opacity/masks/Lung_Opacity-4980.png,True,False,299,299,L,53.13,dfb06765d3d4201774e77cd89e122ef2
Lung_Opacity-4981.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4981.png,Lung_Opacity/images/Lung_Opacity-4981.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4981.png,Lung_Opacity/masks/Lung_Opacity-4981.png,True,False,299,299,L,139.33,3244fc293f9649fa864795aab71a25e6
Lung_Opacity-4982.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4982.png,Lung_Opacity/images/Lung_Opacity-4982.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4982.png,Lung_Opacity/masks/Lung_Opacity-4982.png,True,False,299,299,L,151.15,f08b4cec74da016137bbdcde1bfc96ec
Lung_Opacity-4983.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4983.png,Lung_Opacity/images/Lung_Opacity-4983.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4983.png,Lung_Opacity/masks/Lung_Opacity-4983.png,True,False,299,299,L,130.07,c0acb73503e9d77d5570aaeca0bc0bde
Lung_Opacity-4984.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4984.png,Lung_Opacity/images/Lung_Opacity-4984.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4984.png,Lung_Opacity/masks/Lung_Opacity-4984.png,True,False,299,299,L,159.83,cfbff810fd765f44579a74581101ea95
Lung_Opacity-4985.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4985.png,Lung_Opacity/images/Lung_Opacity-4985.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4985.png,Lung_Opacity/masks/Lung_Opacity-4985.png,True,False,299,299,L,101.96,b73f0100757e64e3f7c6120642bde936
Lung_Opacity-4986.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4986.png,Lung_Opacity/images/Lung_Opacity-4986.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4986.png,Lung_Opacity/masks/Lung_Opacity-4986.png,True,False,299,299,L,96.39,f12486eb4b5c6b1eb9adf4d3febcdd91
Lung_Opacity-4987.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4987.png,Lung_Opacity/images/Lung_Opacity-4987.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4987.png,Lung_Opacity/masks/Lung_Opacity-4987.png,True,False,299,299,L,117.61,fbd69243050c151ac6e84b993c6f88f2
Lung_Opacity-4988.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4988.png,Lung_Opacity/images/Lung_Opacity-4988.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4988.png,Lung_Opacity/masks/Lung_Opacity-4988.png,True,False,299,299,L,156.54,85c8b04c7f63d44b6204063534c63fef
Lung_Opacity-4989.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4989.png,Lung_Opacity/images/Lung_Opacity-4989.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4989.png,Lung_Opacity/masks/Lung_Opacity-4989.png,True,False,299,299,L,118.1,c2f56ea9524a59ff6e289c0bb7b9e710
Lung_Opacity-499.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-499.png,Lung_Opacity/images/Lung_Opacity-499.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-499.png,Lung_Opacity/masks/Lung_Opacity-499.png,True,False,299,299,L,81.7,a0e654cd386117cd70ba1574e6dc3506
Lung_Opacity-4990.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4990.png,Lung_Opacity/images/Lung_Opacity-4990.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4990.png,Lung_Opacity/masks/Lung_Opacity-4990.png,True,False,299,299,L,132.4,327e91e6fe66fe0969f2a05b821fc269
Lung_Opacity-4991.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4991.png,Lung_Opacity/images/Lung_Opacity-4991.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4991.png,Lung_Opacity/masks/Lung_Opacity-4991.png,True,False,299,299,L,117.44,a5af4b5ee2253f3bbe29238c020e30af
Lung_Opacity-4992.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4992.png,Lung_Opacity/images/Lung_Opacity-4992.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4992.png,Lung_Opacity/masks/Lung_Opacity-4992.png,True,False,299,299,L,150.94,29d40af375087a57412db463a5e0f0aa
Lung_Opacity-4993.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4993.png,Lung_Opacity/images/Lung_Opacity-4993.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4993.png,Lung_Opacity/masks/Lung_Opacity-4993.png,True,False,299,299,L,107.56,9736d607aefd9aba92d3031cd2430c70
Lung_Opacity-4994.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4994.png,Lung_Opacity/images/Lung_Opacity-4994.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4994.png,Lung_Opacity/masks/Lung_Opacity-4994.png,True,False,299,299,L,152.4,5e24acc30c5547f717dc6b943dc318f4
Lung_Opacity-4995.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4995.png,Lung_Opacity/images/Lung_Opacity-4995.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4995.png,Lung_Opacity/masks/Lung_Opacity-4995.png,True,False,299,299,L,131.3,a83ec4c2d23fdd07ea61b17fbfdc392e
Lung_Opacity-4996.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4996.png,Lung_Opacity/images/Lung_Opacity-4996.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4996.png,Lung_Opacity/masks/Lung_Opacity-4996.png,True,False,299,299,L,153.05,06f206299efabed7b9586e3f25ee2abe
Lung_Opacity-4997.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4997.png,Lung_Opacity/images/Lung_Opacity-4997.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4997.png,Lung_Opacity/masks/Lung_Opacity-4997.png,True,False,299,299,L,114.69,fa19b9ad47a77598593ad583d7a0003d
Lung_Opacity-4998.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4998.png,Lung_Opacity/images/Lung_Opacity-4998.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4998.png,Lung_Opacity/masks/Lung_Opacity-4998.png,True,False,299,299,L,95.5,a2ff939bd8bf17df1a83fecf5aeb755d
Lung_Opacity-4999.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-4999.png,Lung_Opacity/images/Lung_Opacity-4999.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-4999.png,Lung_Opacity/masks/Lung_Opacity-4999.png,True,False,299,299,L,112.94,55e7289874ebd5a25e56c17514e35781
Lung_Opacity-5.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5.png,Lung_Opacity/images/Lung_Opacity-5.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5.png,Lung_Opacity/masks/Lung_Opacity-5.png,True,False,299,299,L,87.38,f70d3d33fadaa31cd340a63ed3cc12cd
Lung_Opacity-50.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-50.png,Lung_Opacity/images/Lung_Opacity-50.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-50.png,Lung_Opacity/masks/Lung_Opacity-50.png,True,False,299,299,L,155.66,5311a1f7761b647bfafc24488f614a22
Lung_Opacity-500.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-500.png,Lung_Opacity/images/Lung_Opacity-500.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-500.png,Lung_Opacity/masks/Lung_Opacity-500.png,True,False,299,299,L,129.49,38ad480ce1d7880a72410f29b93590d8
Lung_Opacity-5000.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5000.png,Lung_Opacity/images/Lung_Opacity-5000.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5000.png,Lung_Opacity/masks/Lung_Opacity-5000.png,True,False,299,299,L,80.75,53dc1caad11be712d02d5e0fd6581cb9
Lung_Opacity-5001.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5001.png,Lung_Opacity/images/Lung_Opacity-5001.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5001.png,Lung_Opacity/masks/Lung_Opacity-5001.png,True,False,299,299,L,120.43,c066ccf214a0d2b07c818710061e07ff
Lung_Opacity-5002.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5002.png,Lung_Opacity/images/Lung_Opacity-5002.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5002.png,Lung_Opacity/masks/Lung_Opacity-5002.png,True,False,299,299,L,122.17,72b892ce4f3b814a1e8cc2c74491fe4e
Lung_Opacity-5003.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5003.png,Lung_Opacity/images/Lung_Opacity-5003.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5003.png,Lung_Opacity/masks/Lung_Opacity-5003.png,True,False,299,299,L,121.72,dda6e281757150a38ca293d2e94737e9
Lung_Opacity-5004.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5004.png,Lung_Opacity/images/Lung_Opacity-5004.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5004.png,Lung_Opacity/masks/Lung_Opacity-5004.png,True,False,299,299,L,110.73,7c0553614cb746ba319ee39bf7b3b3ac
Lung_Opacity-5005.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5005.png,Lung_Opacity/images/Lung_Opacity-5005.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5005.png,Lung_Opacity/masks/Lung_Opacity-5005.png,True,False,299,299,L,112.84,40e099563fb516764697c236748adc30
Lung_Opacity-5006.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5006.png,Lung_Opacity/images/Lung_Opacity-5006.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5006.png,Lung_Opacity/masks/Lung_Opacity-5006.png,True,False,299,299,L,107.56,3735b37c26a4b3c4af48781e2c744f37
Lung_Opacity-5007.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5007.png,Lung_Opacity/images/Lung_Opacity-5007.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5007.png,Lung_Opacity/masks/Lung_Opacity-5007.png,True,False,299,299,L,120.13,44b96ecc6693232498dc8e8f4a6571d6
Lung_Opacity-5008.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5008.png,Lung_Opacity/images/Lung_Opacity-5008.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5008.png,Lung_Opacity/masks/Lung_Opacity-5008.png,True,False,299,299,L,93.65,dfb21df4f00ba18398c12517d4d36547
Lung_Opacity-5009.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5009.png,Lung_Opacity/images/Lung_Opacity-5009.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5009.png,Lung_Opacity/masks/Lung_Opacity-5009.png,True,False,299,299,L,134.96,f9282de2b5719ec61cbd85c7f6f4218a
Lung_Opacity-501.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-501.png,Lung_Opacity/images/Lung_Opacity-501.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-501.png,Lung_Opacity/masks/Lung_Opacity-501.png,True,False,299,299,L,106.02,6b7690971c26bc8f645db6ab1bf98280
Lung_Opacity-5010.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5010.png,Lung_Opacity/images/Lung_Opacity-5010.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5010.png,Lung_Opacity/masks/Lung_Opacity-5010.png,True,False,299,299,L,100.19,a3528c0f4e5ad97e7c65ab2385a33cb5
Lung_Opacity-5011.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5011.png,Lung_Opacity/images/Lung_Opacity-5011.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5011.png,Lung_Opacity/masks/Lung_Opacity-5011.png,True,False,299,299,L,113.49,8e9e6640d3a4287e8a416c9f4da5d138
Lung_Opacity-5012.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5012.png,Lung_Opacity/images/Lung_Opacity-5012.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5012.png,Lung_Opacity/masks/Lung_Opacity-5012.png,True,False,299,299,L,118.64,00124ca6dc66ad0c499ad4efe3bfd9ea
Lung_Opacity-5013.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5013.png,Lung_Opacity/images/Lung_Opacity-5013.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5013.png,Lung_Opacity/masks/Lung_Opacity-5013.png,True,False,299,299,L,144.19,348ae41f333af279ccd3c52901762bbc
Lung_Opacity-5014.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5014.png,Lung_Opacity/images/Lung_Opacity-5014.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5014.png,Lung_Opacity/masks/Lung_Opacity-5014.png,True,False,299,299,L,151.91,979c5fa41666a5bfd94f9c957737ba35
Lung_Opacity-5015.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5015.png,Lung_Opacity/images/Lung_Opacity-5015.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5015.png,Lung_Opacity/masks/Lung_Opacity-5015.png,True,False,299,299,L,98.64,f316b6ab4346ec8ac92e58b8c9c1517d
Lung_Opacity-5016.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5016.png,Lung_Opacity/images/Lung_Opacity-5016.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5016.png,Lung_Opacity/masks/Lung_Opacity-5016.png,True,False,299,299,L,128.44,291247462cb45020d0ae9be32ca70518
Lung_Opacity-5017.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5017.png,Lung_Opacity/images/Lung_Opacity-5017.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5017.png,Lung_Opacity/masks/Lung_Opacity-5017.png,True,False,299,299,L,121.66,bcc01ee973098c26c435926071ec0e26
Lung_Opacity-5018.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5018.png,Lung_Opacity/images/Lung_Opacity-5018.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5018.png,Lung_Opacity/masks/Lung_Opacity-5018.png,True,False,299,299,L,102.88,4636476c8cf2520b9c234b5691be458f
Lung_Opacity-5019.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5019.png,Lung_Opacity/images/Lung_Opacity-5019.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5019.png,Lung_Opacity/masks/Lung_Opacity-5019.png,True,False,299,299,L,148.82,5f76000f190f7a6afdfda257f80260d1
Lung_Opacity-502.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-502.png,Lung_Opacity/images/Lung_Opacity-502.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-502.png,Lung_Opacity/masks/Lung_Opacity-502.png,True,False,299,299,L,112.27,91e3fac104e211fce8ca1d005dd3be46
Lung_Opacity-5020.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5020.png,Lung_Opacity/images/Lung_Opacity-5020.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5020.png,Lung_Opacity/masks/Lung_Opacity-5020.png,True,False,299,299,L,109.21,47855cf78bc8e0423212dca5b74e282b
Lung_Opacity-5021.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5021.png,Lung_Opacity/images/Lung_Opacity-5021.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5021.png,Lung_Opacity/masks/Lung_Opacity-5021.png,True,False,299,299,L,137.99,bb45dfaa27ba9a45a7877710a7961ca5
Lung_Opacity-5022.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5022.png,Lung_Opacity/images/Lung_Opacity-5022.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5022.png,Lung_Opacity/masks/Lung_Opacity-5022.png,True,False,299,299,L,117.49,f24efd4b38cda5f3bab3c9e3fbeb3ee6
Lung_Opacity-5023.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5023.png,Lung_Opacity/images/Lung_Opacity-5023.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5023.png,Lung_Opacity/masks/Lung_Opacity-5023.png,True,False,299,299,L,143.4,927e9333a0ed0cd1d7865b2ccf81c37c
Lung_Opacity-5024.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5024.png,Lung_Opacity/images/Lung_Opacity-5024.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5024.png,Lung_Opacity/masks/Lung_Opacity-5024.png,True,False,299,299,L,172.77,25d559a21d5e89365f02fa78a89fac15
Lung_Opacity-5025.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5025.png,Lung_Opacity/images/Lung_Opacity-5025.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5025.png,Lung_Opacity/masks/Lung_Opacity-5025.png,True,False,299,299,L,109.94,f3144aa08b68d4cfdeae847667d2d11c
Lung_Opacity-5026.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5026.png,Lung_Opacity/images/Lung_Opacity-5026.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5026.png,Lung_Opacity/masks/Lung_Opacity-5026.png,True,False,299,299,L,134.76,6637186c842ec78602aa508dc48ba974
Lung_Opacity-5027.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5027.png,Lung_Opacity/images/Lung_Opacity-5027.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5027.png,Lung_Opacity/masks/Lung_Opacity-5027.png,True,False,299,299,L,131.59,7da799a6ee37d06b4f3756935bb976d6
Lung_Opacity-5028.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5028.png,Lung_Opacity/images/Lung_Opacity-5028.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5028.png,Lung_Opacity/masks/Lung_Opacity-5028.png,True,False,299,299,L,120.31,07cba84dc7699d7bd1433ad164b835e3
Lung_Opacity-5029.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5029.png,Lung_Opacity/images/Lung_Opacity-5029.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5029.png,Lung_Opacity/masks/Lung_Opacity-5029.png,True,False,299,299,L,114.62,56c533a5b40e3a3ca45b7bbea2377cb4
Lung_Opacity-503.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-503.png,Lung_Opacity/images/Lung_Opacity-503.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-503.png,Lung_Opacity/masks/Lung_Opacity-503.png,True,False,299,299,L,115.05,9caea5141c3a4b840ed84298e0c4b195
Lung_Opacity-5030.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5030.png,Lung_Opacity/images/Lung_Opacity-5030.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5030.png,Lung_Opacity/masks/Lung_Opacity-5030.png,True,False,299,299,L,135.88,bcdb4dfa95232a3b2d6208144e6d326d
Lung_Opacity-5031.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5031.png,Lung_Opacity/images/Lung_Opacity-5031.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5031.png,Lung_Opacity/masks/Lung_Opacity-5031.png,True,False,299,299,L,119.31,454c530cdc968df728ebc4bfda963a20
Lung_Opacity-5032.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5032.png,Lung_Opacity/images/Lung_Opacity-5032.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5032.png,Lung_Opacity/masks/Lung_Opacity-5032.png,True,False,299,299,L,99.99,d8c628f46135dc37b083c7ce7409f631
Lung_Opacity-5033.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5033.png,Lung_Opacity/images/Lung_Opacity-5033.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5033.png,Lung_Opacity/masks/Lung_Opacity-5033.png,True,False,299,299,L,135.37,b54051addc509726f64b05a35340b994
Lung_Opacity-5034.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5034.png,Lung_Opacity/images/Lung_Opacity-5034.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5034.png,Lung_Opacity/masks/Lung_Opacity-5034.png,True,False,299,299,L,133.81,d787007e8de183740857bf9d0396db72
Lung_Opacity-5035.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5035.png,Lung_Opacity/images/Lung_Opacity-5035.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5035.png,Lung_Opacity/masks/Lung_Opacity-5035.png,True,False,299,299,L,120.88,272e57ecc6f7b816a3652ad9ea573d3d
Lung_Opacity-5036.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5036.png,Lung_Opacity/images/Lung_Opacity-5036.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5036.png,Lung_Opacity/masks/Lung_Opacity-5036.png,True,False,299,299,L,125.51,62a57d534829d01c875bfea4ca7c57d1
Lung_Opacity-5037.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5037.png,Lung_Opacity/images/Lung_Opacity-5037.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5037.png,Lung_Opacity/masks/Lung_Opacity-5037.png,True,False,299,299,L,118.98,8c970b99f4402722986e51fdcd915f46
Lung_Opacity-5038.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5038.png,Lung_Opacity/images/Lung_Opacity-5038.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5038.png,Lung_Opacity/masks/Lung_Opacity-5038.png,True,False,299,299,L,106.78,c6b2fef0425beb22a996151a75a73dad
Lung_Opacity-5039.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5039.png,Lung_Opacity/images/Lung_Opacity-5039.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5039.png,Lung_Opacity/masks/Lung_Opacity-5039.png,True,False,299,299,L,107.92,11a389ef9b27e6a76de7d2c332fef129
Lung_Opacity-504.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-504.png,Lung_Opacity/images/Lung_Opacity-504.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-504.png,Lung_Opacity/masks/Lung_Opacity-504.png,True,False,299,299,L,162.46,c9cda0dcef66335761280ee8094ff583
Lung_Opacity-5040.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5040.png,Lung_Opacity/images/Lung_Opacity-5040.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5040.png,Lung_Opacity/masks/Lung_Opacity-5040.png,True,False,299,299,L,229.14,3f0400f5aa21a184f96cd730b1234228
Lung_Opacity-5041.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5041.png,Lung_Opacity/images/Lung_Opacity-5041.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5041.png,Lung_Opacity/masks/Lung_Opacity-5041.png,True,False,299,299,L,108.3,bb973694b8f89ee4057188fbce0f3ff2
Lung_Opacity-5042.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5042.png,Lung_Opacity/images/Lung_Opacity-5042.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5042.png,Lung_Opacity/masks/Lung_Opacity-5042.png,True,False,299,299,L,106.29,6d9ae7cef2dcb326b7b0181c90a5d4a1
Lung_Opacity-5043.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5043.png,Lung_Opacity/images/Lung_Opacity-5043.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5043.png,Lung_Opacity/masks/Lung_Opacity-5043.png,True,False,299,299,L,117.15,759bed795651d55772436f9893eb7da8
Lung_Opacity-5044.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5044.png,Lung_Opacity/images/Lung_Opacity-5044.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5044.png,Lung_Opacity/masks/Lung_Opacity-5044.png,True,False,299,299,L,147.78,88ac0a02a71901b55d834af081127e91
Lung_Opacity-5045.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5045.png,Lung_Opacity/images/Lung_Opacity-5045.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5045.png,Lung_Opacity/masks/Lung_Opacity-5045.png,True,False,299,299,L,141.94,3959f8dba7953579268d1626ec7b2d18
Lung_Opacity-5046.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5046.png,Lung_Opacity/images/Lung_Opacity-5046.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5046.png,Lung_Opacity/masks/Lung_Opacity-5046.png,True,False,299,299,L,139.66,415deec31d9b58e567acb26bbd6b9e3f
Lung_Opacity-5047.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5047.png,Lung_Opacity/images/Lung_Opacity-5047.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5047.png,Lung_Opacity/masks/Lung_Opacity-5047.png,True,False,299,299,L,167.94,9a3aa214f8cc2df910d560362ed5a4ff
Lung_Opacity-5048.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5048.png,Lung_Opacity/images/Lung_Opacity-5048.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5048.png,Lung_Opacity/masks/Lung_Opacity-5048.png,True,False,299,299,L,99.69,51be687870a2143464f7c43ed404dd79
Lung_Opacity-5049.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5049.png,Lung_Opacity/images/Lung_Opacity-5049.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5049.png,Lung_Opacity/masks/Lung_Opacity-5049.png,True,False,299,299,L,144.91,b917c528f592fe3d74fc04c141407f2b
Lung_Opacity-505.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-505.png,Lung_Opacity/images/Lung_Opacity-505.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-505.png,Lung_Opacity/masks/Lung_Opacity-505.png,True,False,299,299,L,160.9,9fe3fc02b5da6662424ff4c76ad3fb05
Lung_Opacity-5050.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5050.png,Lung_Opacity/images/Lung_Opacity-5050.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5050.png,Lung_Opacity/masks/Lung_Opacity-5050.png,True,False,299,299,L,108.97,465b466c60a542988ebd78afda224930
Lung_Opacity-5051.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5051.png,Lung_Opacity/images/Lung_Opacity-5051.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5051.png,Lung_Opacity/masks/Lung_Opacity-5051.png,True,False,299,299,L,102.02,3a0ebe66a0c27e029ba6e981e1a35ef8
Lung_Opacity-5052.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5052.png,Lung_Opacity/images/Lung_Opacity-5052.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5052.png,Lung_Opacity/masks/Lung_Opacity-5052.png,True,False,299,299,L,128.8,06307307b75729d5ce810327c3590c27
Lung_Opacity-5053.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5053.png,Lung_Opacity/images/Lung_Opacity-5053.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5053.png,Lung_Opacity/masks/Lung_Opacity-5053.png,True,False,299,299,L,117.99,e957eff8153cc6c33eb364ea717e2b9a
Lung_Opacity-5054.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5054.png,Lung_Opacity/images/Lung_Opacity-5054.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5054.png,Lung_Opacity/masks/Lung_Opacity-5054.png,True,False,299,299,L,154.31,11cfb36eb973d39e2afb36dd517453ce
Lung_Opacity-5055.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5055.png,Lung_Opacity/images/Lung_Opacity-5055.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5055.png,Lung_Opacity/masks/Lung_Opacity-5055.png,True,False,299,299,L,167.09,e0c914067aa041a7bb0642b3286dafeb
Lung_Opacity-5056.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5056.png,Lung_Opacity/images/Lung_Opacity-5056.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5056.png,Lung_Opacity/masks/Lung_Opacity-5056.png,True,False,299,299,L,172.97,9d6f077bc28d4ebba7fec92c20378c45
Lung_Opacity-5057.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5057.png,Lung_Opacity/images/Lung_Opacity-5057.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5057.png,Lung_Opacity/masks/Lung_Opacity-5057.png,True,False,299,299,L,82.99,716d9ba19ab456e271fba5f5ed0c4d2f
Lung_Opacity-5058.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5058.png,Lung_Opacity/images/Lung_Opacity-5058.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5058.png,Lung_Opacity/masks/Lung_Opacity-5058.png,True,False,299,299,L,173.35,5595d9efc2ecb9381a74826b018d394d
Lung_Opacity-5059.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5059.png,Lung_Opacity/images/Lung_Opacity-5059.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5059.png,Lung_Opacity/masks/Lung_Opacity-5059.png,True,False,299,299,L,121.61,3c0ee174943178d91a053dfe2cd71b25
Lung_Opacity-506.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-506.png,Lung_Opacity/images/Lung_Opacity-506.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-506.png,Lung_Opacity/masks/Lung_Opacity-506.png,True,False,299,299,L,138.07,e1fd20f4b347af5a8239bfbaf9e9489f
Lung_Opacity-5060.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5060.png,Lung_Opacity/images/Lung_Opacity-5060.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5060.png,Lung_Opacity/masks/Lung_Opacity-5060.png,True,False,299,299,L,101.61,6924dc8bd432ed43a665b32fdbedf380
Lung_Opacity-5061.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5061.png,Lung_Opacity/images/Lung_Opacity-5061.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5061.png,Lung_Opacity/masks/Lung_Opacity-5061.png,True,False,299,299,L,123.32,ac54feb80a23693d7752da0e4658bd79
Lung_Opacity-5062.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5062.png,Lung_Opacity/images/Lung_Opacity-5062.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5062.png,Lung_Opacity/masks/Lung_Opacity-5062.png,True,False,299,299,L,164.27,8aa64d01ee599c6d261d01638471b77a
Lung_Opacity-5063.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5063.png,Lung_Opacity/images/Lung_Opacity-5063.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5063.png,Lung_Opacity/masks/Lung_Opacity-5063.png,True,False,299,299,L,109.62,d744391dc434883a8c108730ffc0321c
Lung_Opacity-5064.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5064.png,Lung_Opacity/images/Lung_Opacity-5064.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5064.png,Lung_Opacity/masks/Lung_Opacity-5064.png,True,False,299,299,L,107.02,98dc0359be41f00a13c361134e220f86
Lung_Opacity-5065.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5065.png,Lung_Opacity/images/Lung_Opacity-5065.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5065.png,Lung_Opacity/masks/Lung_Opacity-5065.png,True,False,299,299,L,120.72,fa872e250794bfa928375f02ffb0b1e4
Lung_Opacity-5066.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5066.png,Lung_Opacity/images/Lung_Opacity-5066.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5066.png,Lung_Opacity/masks/Lung_Opacity-5066.png,True,False,299,299,L,126.22,e29fcfc8ba5d3ed65ee76910baea56b8
Lung_Opacity-5067.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5067.png,Lung_Opacity/images/Lung_Opacity-5067.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5067.png,Lung_Opacity/masks/Lung_Opacity-5067.png,True,False,299,299,L,112.6,1d3a6691812653554a584f7c35469abe
Lung_Opacity-5068.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5068.png,Lung_Opacity/images/Lung_Opacity-5068.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5068.png,Lung_Opacity/masks/Lung_Opacity-5068.png,True,False,299,299,L,111.14,59266282c6111ea9c6a0569647b732f8
Lung_Opacity-5069.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5069.png,Lung_Opacity/images/Lung_Opacity-5069.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5069.png,Lung_Opacity/masks/Lung_Opacity-5069.png,True,False,299,299,L,41.54,24c9d8b5ca3e84f619c0fdfcdbff89f3
Lung_Opacity-507.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-507.png,Lung_Opacity/images/Lung_Opacity-507.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-507.png,Lung_Opacity/masks/Lung_Opacity-507.png,True,False,299,299,L,134.45,599f4df211806a0e445ca2b2e8b84ad8
Lung_Opacity-5070.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5070.png,Lung_Opacity/images/Lung_Opacity-5070.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5070.png,Lung_Opacity/masks/Lung_Opacity-5070.png,True,False,299,299,L,124.62,b09c4937974337f428a7aeac0b51aa70
Lung_Opacity-5071.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5071.png,Lung_Opacity/images/Lung_Opacity-5071.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5071.png,Lung_Opacity/masks/Lung_Opacity-5071.png,True,False,299,299,L,115.65,5ba0cb91546f7601fc070eb853661225
Lung_Opacity-5072.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5072.png,Lung_Opacity/images/Lung_Opacity-5072.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5072.png,Lung_Opacity/masks/Lung_Opacity-5072.png,True,False,299,299,L,118.94,75cf5d34a626ad38730b465d426bfad1
Lung_Opacity-5073.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5073.png,Lung_Opacity/images/Lung_Opacity-5073.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5073.png,Lung_Opacity/masks/Lung_Opacity-5073.png,True,False,299,299,L,122.35,7cc0fadbc68c7158a9b2191b5e4823e4
Lung_Opacity-5074.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5074.png,Lung_Opacity/images/Lung_Opacity-5074.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5074.png,Lung_Opacity/masks/Lung_Opacity-5074.png,True,False,299,299,L,176.66,ade56c4311af6ba620d4158ca1ee4968
Lung_Opacity-5075.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5075.png,Lung_Opacity/images/Lung_Opacity-5075.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5075.png,Lung_Opacity/masks/Lung_Opacity-5075.png,True,False,299,299,L,117.25,272ba46e3a1df2c22e772e42b6dd98c6
Lung_Opacity-5076.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5076.png,Lung_Opacity/images/Lung_Opacity-5076.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5076.png,Lung_Opacity/masks/Lung_Opacity-5076.png,True,False,299,299,L,117.98,df7c1068966bbcf32ea6e17c39f35119
Lung_Opacity-5077.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5077.png,Lung_Opacity/images/Lung_Opacity-5077.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5077.png,Lung_Opacity/masks/Lung_Opacity-5077.png,True,False,299,299,L,100.3,ad0c22be0258854a382cdfac1f49772f
Lung_Opacity-5078.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5078.png,Lung_Opacity/images/Lung_Opacity-5078.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5078.png,Lung_Opacity/masks/Lung_Opacity-5078.png,True,False,299,299,L,102.43,3405be66fd8f8d0a156f35d2fa856252
Lung_Opacity-5079.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5079.png,Lung_Opacity/images/Lung_Opacity-5079.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5079.png,Lung_Opacity/masks/Lung_Opacity-5079.png,True,False,299,299,L,167.64,5dae131df0a255bbac43309d69711ad0
Lung_Opacity-508.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-508.png,Lung_Opacity/images/Lung_Opacity-508.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-508.png,Lung_Opacity/masks/Lung_Opacity-508.png,True,False,299,299,L,117.67,a8a9eae051aca9e65c25414ea41f765f
Lung_Opacity-5080.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5080.png,Lung_Opacity/images/Lung_Opacity-5080.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5080.png,Lung_Opacity/masks/Lung_Opacity-5080.png,True,False,299,299,L,146.8,f6e91151ef5b702e1a4050e066f64123
Lung_Opacity-5081.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5081.png,Lung_Opacity/images/Lung_Opacity-5081.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5081.png,Lung_Opacity/masks/Lung_Opacity-5081.png,True,False,299,299,L,154.71,bd4534935bcacebefa26790ea26b4f2a
Lung_Opacity-5082.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5082.png,Lung_Opacity/images/Lung_Opacity-5082.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5082.png,Lung_Opacity/masks/Lung_Opacity-5082.png,True,False,299,299,L,153.99,ec827a2cc5c1064d3e7c6de232330450
Lung_Opacity-5083.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5083.png,Lung_Opacity/images/Lung_Opacity-5083.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5083.png,Lung_Opacity/masks/Lung_Opacity-5083.png,True,False,299,299,L,107.59,fa3d0f75d9b3e18c8766fd2bc523f37b
Lung_Opacity-5084.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5084.png,Lung_Opacity/images/Lung_Opacity-5084.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5084.png,Lung_Opacity/masks/Lung_Opacity-5084.png,True,False,299,299,L,113.67,c09192aad5eee0ddda7782b35d2fc466
Lung_Opacity-5085.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5085.png,Lung_Opacity/images/Lung_Opacity-5085.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5085.png,Lung_Opacity/masks/Lung_Opacity-5085.png,True,False,299,299,L,112.94,0bf44306fd38fc1b6327e2479a6f293e
Lung_Opacity-5086.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5086.png,Lung_Opacity/images/Lung_Opacity-5086.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5086.png,Lung_Opacity/masks/Lung_Opacity-5086.png,True,False,299,299,L,174.73,8e1e9bac7990a6b84199cf7e357f3283
Lung_Opacity-5087.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5087.png,Lung_Opacity/images/Lung_Opacity-5087.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5087.png,Lung_Opacity/masks/Lung_Opacity-5087.png,True,False,299,299,L,112.57,dc112692ff9ddd119f2ada89925a2658
Lung_Opacity-5088.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5088.png,Lung_Opacity/images/Lung_Opacity-5088.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5088.png,Lung_Opacity/masks/Lung_Opacity-5088.png,True,False,299,299,L,133.88,87fbcb2dcf9a565e71ac87df01db79ab
Lung_Opacity-5089.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5089.png,Lung_Opacity/images/Lung_Opacity-5089.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5089.png,Lung_Opacity/masks/Lung_Opacity-5089.png,True,False,299,299,L,102.62,742beecb142867e897c548509c43078e
Lung_Opacity-509.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-509.png,Lung_Opacity/images/Lung_Opacity-509.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-509.png,Lung_Opacity/masks/Lung_Opacity-509.png,True,False,299,299,L,110.33,fd73a807e35221b61b50abfa73b628d2
Lung_Opacity-5090.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5090.png,Lung_Opacity/images/Lung_Opacity-5090.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5090.png,Lung_Opacity/masks/Lung_Opacity-5090.png,True,False,299,299,L,112.6,50020fb79e6cf9a5ce98d6310c3be227
Lung_Opacity-5091.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5091.png,Lung_Opacity/images/Lung_Opacity-5091.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5091.png,Lung_Opacity/masks/Lung_Opacity-5091.png,True,False,299,299,L,84.69,6bdd851596cd15e2e09246bc91922bef
Lung_Opacity-5092.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5092.png,Lung_Opacity/images/Lung_Opacity-5092.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5092.png,Lung_Opacity/masks/Lung_Opacity-5092.png,True,False,299,299,L,69.39,1051f5ce6a6b8efbd4ead01c8fbf52bd
Lung_Opacity-5093.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5093.png,Lung_Opacity/images/Lung_Opacity-5093.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5093.png,Lung_Opacity/masks/Lung_Opacity-5093.png,True,False,299,299,L,145.29,3ef81b48633a22ef4b6f6d0073bdcaf2
Lung_Opacity-5094.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5094.png,Lung_Opacity/images/Lung_Opacity-5094.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5094.png,Lung_Opacity/masks/Lung_Opacity-5094.png,True,False,299,299,L,108.17,d4bdbf513e829b9389c7545dac17159d
Lung_Opacity-5095.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5095.png,Lung_Opacity/images/Lung_Opacity-5095.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5095.png,Lung_Opacity/masks/Lung_Opacity-5095.png,True,False,299,299,L,137.56,1e1d85a69ebe1d199dd10b78877d7574
Lung_Opacity-5096.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5096.png,Lung_Opacity/images/Lung_Opacity-5096.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5096.png,Lung_Opacity/masks/Lung_Opacity-5096.png,True,False,299,299,L,105.22,6c06f34707c5557d9d205a99e3381429
Lung_Opacity-5097.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5097.png,Lung_Opacity/images/Lung_Opacity-5097.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5097.png,Lung_Opacity/masks/Lung_Opacity-5097.png,True,False,299,299,L,116.8,a9b2acc8ebc595083e2b8af59c3b29a2
Lung_Opacity-5098.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5098.png,Lung_Opacity/images/Lung_Opacity-5098.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5098.png,Lung_Opacity/masks/Lung_Opacity-5098.png,True,False,299,299,L,109.88,0e9fe09afc17ac861a1fabd4ebceb8d6
Lung_Opacity-5099.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5099.png,Lung_Opacity/images/Lung_Opacity-5099.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5099.png,Lung_Opacity/masks/Lung_Opacity-5099.png,True,False,299,299,L,129.03,888395573a37f0ebe112ea974ae19599
Lung_Opacity-51.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-51.png,Lung_Opacity/images/Lung_Opacity-51.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-51.png,Lung_Opacity/masks/Lung_Opacity-51.png,True,False,299,299,L,174.53,cdedd40c7b9a2f6f9f99dad42e90986b
Lung_Opacity-510.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-510.png,Lung_Opacity/images/Lung_Opacity-510.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-510.png,Lung_Opacity/masks/Lung_Opacity-510.png,True,False,299,299,L,95.55,6b8112f3b4a44343780adb5e8e5de384
Lung_Opacity-5100.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5100.png,Lung_Opacity/images/Lung_Opacity-5100.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5100.png,Lung_Opacity/masks/Lung_Opacity-5100.png,True,False,299,299,L,111.12,62a180cdbae50307b176c138292db138
Lung_Opacity-5101.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5101.png,Lung_Opacity/images/Lung_Opacity-5101.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5101.png,Lung_Opacity/masks/Lung_Opacity-5101.png,True,False,299,299,L,103.92,1ad573ef6e8ba7ea261f0040d814ce42
Lung_Opacity-5102.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5102.png,Lung_Opacity/images/Lung_Opacity-5102.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5102.png,Lung_Opacity/masks/Lung_Opacity-5102.png,True,False,299,299,L,112.94,2ad764b95faf8c848b667574a161b0b9
Lung_Opacity-5103.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5103.png,Lung_Opacity/images/Lung_Opacity-5103.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5103.png,Lung_Opacity/masks/Lung_Opacity-5103.png,True,False,299,299,L,87.35,cf596b2e6a2fb26e97c8f3db574bb324
Lung_Opacity-5104.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5104.png,Lung_Opacity/images/Lung_Opacity-5104.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5104.png,Lung_Opacity/masks/Lung_Opacity-5104.png,True,False,299,299,L,119.79,45dbe29881eb67d3c2b080cda13b9cbd
Lung_Opacity-5105.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5105.png,Lung_Opacity/images/Lung_Opacity-5105.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5105.png,Lung_Opacity/masks/Lung_Opacity-5105.png,True,False,299,299,L,148.82,f51770f66bfc03fbd7047bceda629457
Lung_Opacity-5106.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5106.png,Lung_Opacity/images/Lung_Opacity-5106.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5106.png,Lung_Opacity/masks/Lung_Opacity-5106.png,True,False,299,299,L,111.48,0c1b77275f9cb39456bf418c25b9b32f
Lung_Opacity-5107.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5107.png,Lung_Opacity/images/Lung_Opacity-5107.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5107.png,Lung_Opacity/masks/Lung_Opacity-5107.png,True,False,299,299,L,126.06,ee97d268211dedb2deec71db8738eb23
Lung_Opacity-5108.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5108.png,Lung_Opacity/images/Lung_Opacity-5108.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5108.png,Lung_Opacity/masks/Lung_Opacity-5108.png,True,False,299,299,L,110.03,1c4e7ea0ebac2298a76b24237ddd0c86
Lung_Opacity-5109.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5109.png,Lung_Opacity/images/Lung_Opacity-5109.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5109.png,Lung_Opacity/masks/Lung_Opacity-5109.png,True,False,299,299,L,113.32,e44d5ec2a27aaa4fd06862293a4dafa5
Lung_Opacity-511.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-511.png,Lung_Opacity/images/Lung_Opacity-511.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-511.png,Lung_Opacity/masks/Lung_Opacity-511.png,True,False,299,299,L,130.46,fc1ba2fe54f5c3b3d3952307118cbff1
Lung_Opacity-5110.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5110.png,Lung_Opacity/images/Lung_Opacity-5110.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5110.png,Lung_Opacity/masks/Lung_Opacity-5110.png,True,False,299,299,L,147.51,0969d0bee4ee35d514ff986a34f04869
Lung_Opacity-5111.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5111.png,Lung_Opacity/images/Lung_Opacity-5111.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5111.png,Lung_Opacity/masks/Lung_Opacity-5111.png,True,False,299,299,L,168.34,100f2b95dd27a803cebbea334bd22913
Lung_Opacity-5112.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5112.png,Lung_Opacity/images/Lung_Opacity-5112.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5112.png,Lung_Opacity/masks/Lung_Opacity-5112.png,True,False,299,299,L,120.07,8be5dc7be99cab4e3da895c4a6566b35
Lung_Opacity-5113.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5113.png,Lung_Opacity/images/Lung_Opacity-5113.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5113.png,Lung_Opacity/masks/Lung_Opacity-5113.png,True,False,299,299,L,98.71,25224b981999e0a3d7c8af8002f2f623
Lung_Opacity-5114.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5114.png,Lung_Opacity/images/Lung_Opacity-5114.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5114.png,Lung_Opacity/masks/Lung_Opacity-5114.png,True,False,299,299,L,162.57,6604203f11a056c2994ed281863ffd1b
Lung_Opacity-5115.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5115.png,Lung_Opacity/images/Lung_Opacity-5115.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5115.png,Lung_Opacity/masks/Lung_Opacity-5115.png,True,False,299,299,L,77.9,1092ecb560c5076a668b6ef20a3e6658
Lung_Opacity-5116.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5116.png,Lung_Opacity/images/Lung_Opacity-5116.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5116.png,Lung_Opacity/masks/Lung_Opacity-5116.png,True,False,299,299,L,131.54,b5f7b4f1d0a51757f78f8dbb48e94a10
Lung_Opacity-5117.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5117.png,Lung_Opacity/images/Lung_Opacity-5117.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5117.png,Lung_Opacity/masks/Lung_Opacity-5117.png,True,False,299,299,L,109.16,e0cbaed3b00819ae7eecccf1ae5d20db
Lung_Opacity-5118.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5118.png,Lung_Opacity/images/Lung_Opacity-5118.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5118.png,Lung_Opacity/masks/Lung_Opacity-5118.png,True,False,299,299,L,120.6,bc29169081dfa73cec89043f58f0c7a0
Lung_Opacity-5119.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5119.png,Lung_Opacity/images/Lung_Opacity-5119.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5119.png,Lung_Opacity/masks/Lung_Opacity-5119.png,True,False,299,299,L,115.05,8fe31c22450e7f02d274c2820e3e427e
Lung_Opacity-512.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-512.png,Lung_Opacity/images/Lung_Opacity-512.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-512.png,Lung_Opacity/masks/Lung_Opacity-512.png,True,False,299,299,L,118.7,5205858fc85ff998ce3436766d9a8a8f
Lung_Opacity-5120.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5120.png,Lung_Opacity/images/Lung_Opacity-5120.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5120.png,Lung_Opacity/masks/Lung_Opacity-5120.png,True,False,299,299,L,114.58,fe8d6264c889abddaa8389a96d3337d1
Lung_Opacity-5121.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5121.png,Lung_Opacity/images/Lung_Opacity-5121.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5121.png,Lung_Opacity/masks/Lung_Opacity-5121.png,True,False,299,299,L,122.02,91d5aa481571db0db3849b92b25c5ff4
Lung_Opacity-5122.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5122.png,Lung_Opacity/images/Lung_Opacity-5122.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5122.png,Lung_Opacity/masks/Lung_Opacity-5122.png,True,False,299,299,L,149.06,8fbdad73673770b731ab135633b3189a
Lung_Opacity-5123.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5123.png,Lung_Opacity/images/Lung_Opacity-5123.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5123.png,Lung_Opacity/masks/Lung_Opacity-5123.png,True,False,299,299,L,149.9,c2d23ea2353bc1b742f1e7c749827b56
Lung_Opacity-5124.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5124.png,Lung_Opacity/images/Lung_Opacity-5124.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5124.png,Lung_Opacity/masks/Lung_Opacity-5124.png,True,False,299,299,L,109.42,17dca057761908756f70302a0636527b
Lung_Opacity-5125.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5125.png,Lung_Opacity/images/Lung_Opacity-5125.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5125.png,Lung_Opacity/masks/Lung_Opacity-5125.png,True,False,299,299,L,115.31,d2f3b2602c74fdd491a75eaf40b5863c
Lung_Opacity-5126.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5126.png,Lung_Opacity/images/Lung_Opacity-5126.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5126.png,Lung_Opacity/masks/Lung_Opacity-5126.png,True,False,299,299,L,131.54,f90da708a8183bc38348cbac9ab6b60a
Lung_Opacity-5127.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5127.png,Lung_Opacity/images/Lung_Opacity-5127.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5127.png,Lung_Opacity/masks/Lung_Opacity-5127.png,True,False,299,299,L,169.92,6954ebed524e11d3f033ed6defdc49d7
Lung_Opacity-5128.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5128.png,Lung_Opacity/images/Lung_Opacity-5128.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5128.png,Lung_Opacity/masks/Lung_Opacity-5128.png,True,False,299,299,L,114.89,02d49a82179c4e829bda2ba0c4656aba
Lung_Opacity-5129.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5129.png,Lung_Opacity/images/Lung_Opacity-5129.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5129.png,Lung_Opacity/masks/Lung_Opacity-5129.png,True,False,299,299,L,114.45,1b67c25a87c4f3c7f048c897416f7ace
Lung_Opacity-513.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-513.png,Lung_Opacity/images/Lung_Opacity-513.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-513.png,Lung_Opacity/masks/Lung_Opacity-513.png,True,False,299,299,L,162.31,ed9be0fc3414aadf8836454fa812cbb7
Lung_Opacity-5130.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5130.png,Lung_Opacity/images/Lung_Opacity-5130.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5130.png,Lung_Opacity/masks/Lung_Opacity-5130.png,True,False,299,299,L,112.89,71ba257fd6520b928d43ea1d251076bd
Lung_Opacity-5131.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5131.png,Lung_Opacity/images/Lung_Opacity-5131.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5131.png,Lung_Opacity/masks/Lung_Opacity-5131.png,True,False,299,299,L,149.43,3b5ce8ab99dcef3d7d8f1493522fd076
Lung_Opacity-5132.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5132.png,Lung_Opacity/images/Lung_Opacity-5132.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5132.png,Lung_Opacity/masks/Lung_Opacity-5132.png,True,False,299,299,L,152.82,b6e61eeae726ff0660f18eedba50c75f
Lung_Opacity-5133.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5133.png,Lung_Opacity/images/Lung_Opacity-5133.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5133.png,Lung_Opacity/masks/Lung_Opacity-5133.png,True,False,299,299,L,115.54,ddcaffcfba7808191cfac52859d29abe
Lung_Opacity-5134.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5134.png,Lung_Opacity/images/Lung_Opacity-5134.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5134.png,Lung_Opacity/masks/Lung_Opacity-5134.png,True,False,299,299,L,112.63,1f7ad6e7d1f718498e57e4cf5d7817a8
Lung_Opacity-5135.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5135.png,Lung_Opacity/images/Lung_Opacity-5135.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5135.png,Lung_Opacity/masks/Lung_Opacity-5135.png,True,False,299,299,L,139.28,c85d251667769d673eb3cd0b9bc743a9
Lung_Opacity-5136.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5136.png,Lung_Opacity/images/Lung_Opacity-5136.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5136.png,Lung_Opacity/masks/Lung_Opacity-5136.png,True,False,299,299,L,101.66,5ecabc18f85fab7ce6458d7c3a84dce5
Lung_Opacity-5137.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5137.png,Lung_Opacity/images/Lung_Opacity-5137.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5137.png,Lung_Opacity/masks/Lung_Opacity-5137.png,True,False,299,299,L,110.58,052de4ab5fd8ef75ab795c0945b812ad
Lung_Opacity-5138.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5138.png,Lung_Opacity/images/Lung_Opacity-5138.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5138.png,Lung_Opacity/masks/Lung_Opacity-5138.png,True,False,299,299,L,51.51,28601743a237c8e2c166b1d50ce74e26
Lung_Opacity-5139.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5139.png,Lung_Opacity/images/Lung_Opacity-5139.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5139.png,Lung_Opacity/masks/Lung_Opacity-5139.png,True,False,299,299,L,92.11,b7b8b8cf993c2d1ca8acfd55f04a583b
Lung_Opacity-514.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-514.png,Lung_Opacity/images/Lung_Opacity-514.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-514.png,Lung_Opacity/masks/Lung_Opacity-514.png,True,False,299,299,L,114.01,db38f288091b808f72e16e21700ebaee
Lung_Opacity-5140.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5140.png,Lung_Opacity/images/Lung_Opacity-5140.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5140.png,Lung_Opacity/masks/Lung_Opacity-5140.png,True,False,299,299,L,119.78,b061f9f300517871fd7aaa39d83c0b32
Lung_Opacity-5141.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5141.png,Lung_Opacity/images/Lung_Opacity-5141.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5141.png,Lung_Opacity/masks/Lung_Opacity-5141.png,True,False,299,299,L,115.67,251ae4634f20b6ccb7aacb77e0fd9abd
Lung_Opacity-5142.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5142.png,Lung_Opacity/images/Lung_Opacity-5142.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5142.png,Lung_Opacity/masks/Lung_Opacity-5142.png,True,False,299,299,L,70.5,524cae239674b071a65fe09c17f922b3
Lung_Opacity-5143.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5143.png,Lung_Opacity/images/Lung_Opacity-5143.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5143.png,Lung_Opacity/masks/Lung_Opacity-5143.png,True,False,299,299,L,101.62,e0962a5b440d0f19c2bcc0fe14708f5d
Lung_Opacity-5144.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5144.png,Lung_Opacity/images/Lung_Opacity-5144.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5144.png,Lung_Opacity/masks/Lung_Opacity-5144.png,True,False,299,299,L,115.78,588d08548667d327e9cc6e2fd130c906
Lung_Opacity-5145.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5145.png,Lung_Opacity/images/Lung_Opacity-5145.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5145.png,Lung_Opacity/masks/Lung_Opacity-5145.png,True,False,299,299,L,127.72,517e7e04ffc5a81d986cd5bc4c142010
Lung_Opacity-5146.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5146.png,Lung_Opacity/images/Lung_Opacity-5146.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5146.png,Lung_Opacity/masks/Lung_Opacity-5146.png,True,False,299,299,L,125.79,7710f8d6bda8e1af08e8202067955de9
Lung_Opacity-5147.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5147.png,Lung_Opacity/images/Lung_Opacity-5147.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5147.png,Lung_Opacity/masks/Lung_Opacity-5147.png,True,False,299,299,L,123.21,127c79ca6ab2f6c6e23bd728f65ea99d
Lung_Opacity-5148.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5148.png,Lung_Opacity/images/Lung_Opacity-5148.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5148.png,Lung_Opacity/masks/Lung_Opacity-5148.png,True,False,299,299,L,118.1,df9027e222ad4469a3d3d8cdf4d01b5b
Lung_Opacity-5149.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5149.png,Lung_Opacity/images/Lung_Opacity-5149.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5149.png,Lung_Opacity/masks/Lung_Opacity-5149.png,True,False,299,299,L,88.65,01044e6be2a2fed96069bdefc9f8dbbd
Lung_Opacity-515.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-515.png,Lung_Opacity/images/Lung_Opacity-515.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-515.png,Lung_Opacity/masks/Lung_Opacity-515.png,True,False,299,299,L,111.72,a21cf8abc6a5e06b4fc8b604d670d13e
Lung_Opacity-5150.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5150.png,Lung_Opacity/images/Lung_Opacity-5150.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5150.png,Lung_Opacity/masks/Lung_Opacity-5150.png,True,False,299,299,L,110.28,f2d39b972bea56e864f9b75acd3f3395
Lung_Opacity-5151.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5151.png,Lung_Opacity/images/Lung_Opacity-5151.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5151.png,Lung_Opacity/masks/Lung_Opacity-5151.png,True,False,299,299,L,130.65,6cf0403fa3f761c77be30fa508a9fa4f
Lung_Opacity-5152.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5152.png,Lung_Opacity/images/Lung_Opacity-5152.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5152.png,Lung_Opacity/masks/Lung_Opacity-5152.png,True,False,299,299,L,144.29,191cc8d790b416d80c3a84f3b71cb8b8
Lung_Opacity-5153.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5153.png,Lung_Opacity/images/Lung_Opacity-5153.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5153.png,Lung_Opacity/masks/Lung_Opacity-5153.png,True,False,299,299,L,123.58,580382fe02343f5176e938bd7f96755c
Lung_Opacity-5154.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5154.png,Lung_Opacity/images/Lung_Opacity-5154.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5154.png,Lung_Opacity/masks/Lung_Opacity-5154.png,True,False,299,299,L,108.79,44634d53f3de33eee5b035a81db08b78
Lung_Opacity-5155.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5155.png,Lung_Opacity/images/Lung_Opacity-5155.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5155.png,Lung_Opacity/masks/Lung_Opacity-5155.png,True,False,299,299,L,99.25,2514ddb2357139cf182f810758ed9bf0
Lung_Opacity-5156.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5156.png,Lung_Opacity/images/Lung_Opacity-5156.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5156.png,Lung_Opacity/masks/Lung_Opacity-5156.png,True,False,299,299,L,132.23,70b5a10796cec5e21f94b8880232e52b
Lung_Opacity-5157.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5157.png,Lung_Opacity/images/Lung_Opacity-5157.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5157.png,Lung_Opacity/masks/Lung_Opacity-5157.png,True,False,299,299,L,178.35,0ea81c1771a9ec51f08ede97557165c7
Lung_Opacity-5158.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5158.png,Lung_Opacity/images/Lung_Opacity-5158.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5158.png,Lung_Opacity/masks/Lung_Opacity-5158.png,True,False,299,299,L,106.99,4361940d84b3879c5f29f223a52a3783
Lung_Opacity-5159.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5159.png,Lung_Opacity/images/Lung_Opacity-5159.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5159.png,Lung_Opacity/masks/Lung_Opacity-5159.png,True,False,299,299,L,145.38,1b3a5518ec4b08174a038157483e2962
Lung_Opacity-516.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-516.png,Lung_Opacity/images/Lung_Opacity-516.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-516.png,Lung_Opacity/masks/Lung_Opacity-516.png,True,False,299,299,L,111.47,6ee3ed6c90a74d16c61c3060ed3722a9
Lung_Opacity-5160.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5160.png,Lung_Opacity/images/Lung_Opacity-5160.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5160.png,Lung_Opacity/masks/Lung_Opacity-5160.png,True,False,299,299,L,160.41,fb3ef813c96020481c8242fe9371d10e
Lung_Opacity-5161.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5161.png,Lung_Opacity/images/Lung_Opacity-5161.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5161.png,Lung_Opacity/masks/Lung_Opacity-5161.png,True,False,299,299,L,120.48,95221de83941f19e3027df118448a81f
Lung_Opacity-5162.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5162.png,Lung_Opacity/images/Lung_Opacity-5162.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5162.png,Lung_Opacity/masks/Lung_Opacity-5162.png,True,False,299,299,L,109.25,464006e00e1b41c57fe1338faa81ddf6
Lung_Opacity-5163.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5163.png,Lung_Opacity/images/Lung_Opacity-5163.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5163.png,Lung_Opacity/masks/Lung_Opacity-5163.png,True,False,299,299,L,115.92,4f106766bd9270ccda18f80454e4be76
Lung_Opacity-5164.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5164.png,Lung_Opacity/images/Lung_Opacity-5164.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5164.png,Lung_Opacity/masks/Lung_Opacity-5164.png,True,False,299,299,L,149.26,ace72f820204e0a39e4b6893e3ce5a06
Lung_Opacity-5165.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5165.png,Lung_Opacity/images/Lung_Opacity-5165.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5165.png,Lung_Opacity/masks/Lung_Opacity-5165.png,True,False,299,299,L,128.81,740acd5aa395831411c1f1c1061936b6
Lung_Opacity-5166.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5166.png,Lung_Opacity/images/Lung_Opacity-5166.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5166.png,Lung_Opacity/masks/Lung_Opacity-5166.png,True,False,299,299,L,97.51,5f9ab0f4fab5c196df4238fbda2bf0ed
Lung_Opacity-5167.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5167.png,Lung_Opacity/images/Lung_Opacity-5167.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5167.png,Lung_Opacity/masks/Lung_Opacity-5167.png,True,False,299,299,L,106.67,d0d1c68ad2eea1ec57c161df3ef02997
Lung_Opacity-5168.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5168.png,Lung_Opacity/images/Lung_Opacity-5168.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5168.png,Lung_Opacity/masks/Lung_Opacity-5168.png,True,False,299,299,L,129.24,f14890b654b10a83a9d20e2897103698
Lung_Opacity-5169.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5169.png,Lung_Opacity/images/Lung_Opacity-5169.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5169.png,Lung_Opacity/masks/Lung_Opacity-5169.png,True,False,299,299,L,89.54,0926afa409eb0c6fe32ed3bf31d975f5
Lung_Opacity-517.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-517.png,Lung_Opacity/images/Lung_Opacity-517.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-517.png,Lung_Opacity/masks/Lung_Opacity-517.png,True,False,299,299,L,114.29,80dca2bb01c4cfb2f6df92025c9d2cf4
Lung_Opacity-5170.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5170.png,Lung_Opacity/images/Lung_Opacity-5170.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5170.png,Lung_Opacity/masks/Lung_Opacity-5170.png,True,False,299,299,L,148.13,c72ce37c67a698ca8c6f9815026f0a68
Lung_Opacity-5171.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5171.png,Lung_Opacity/images/Lung_Opacity-5171.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5171.png,Lung_Opacity/masks/Lung_Opacity-5171.png,True,False,299,299,L,108.02,ac9a30c072b07c451716ebc2d6b3b7bc
Lung_Opacity-5172.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5172.png,Lung_Opacity/images/Lung_Opacity-5172.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5172.png,Lung_Opacity/masks/Lung_Opacity-5172.png,True,False,299,299,L,102.19,65e26b6ab47440fb52ae24842d09be83
Lung_Opacity-5173.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5173.png,Lung_Opacity/images/Lung_Opacity-5173.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5173.png,Lung_Opacity/masks/Lung_Opacity-5173.png,True,False,299,299,L,107.16,d508de3f0d169a1a20fd0f2e0a08def9
Lung_Opacity-5174.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5174.png,Lung_Opacity/images/Lung_Opacity-5174.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5174.png,Lung_Opacity/masks/Lung_Opacity-5174.png,True,False,299,299,L,167.49,c398d892124ebdb3dfb1d10b68b1f073
Lung_Opacity-5175.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5175.png,Lung_Opacity/images/Lung_Opacity-5175.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5175.png,Lung_Opacity/masks/Lung_Opacity-5175.png,True,False,299,299,L,119.1,3056bb9f2f18534574a03d18ccc27fc4
Lung_Opacity-5176.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5176.png,Lung_Opacity/images/Lung_Opacity-5176.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5176.png,Lung_Opacity/masks/Lung_Opacity-5176.png,True,False,299,299,L,121.87,56c8a9f7313a5b557f8dccfb255a2992
Lung_Opacity-5177.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5177.png,Lung_Opacity/images/Lung_Opacity-5177.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5177.png,Lung_Opacity/masks/Lung_Opacity-5177.png,True,False,299,299,L,128.91,474a398987a367ca257e18c420a7e28f
Lung_Opacity-5178.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5178.png,Lung_Opacity/images/Lung_Opacity-5178.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5178.png,Lung_Opacity/masks/Lung_Opacity-5178.png,True,False,299,299,L,121.66,52830765513702cc463f0e95d40132bc
Lung_Opacity-5179.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5179.png,Lung_Opacity/images/Lung_Opacity-5179.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5179.png,Lung_Opacity/masks/Lung_Opacity-5179.png,True,False,299,299,L,145.87,bf4e2bf8bdc81dc98fb97be30f255ca7
Lung_Opacity-518.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-518.png,Lung_Opacity/images/Lung_Opacity-518.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-518.png,Lung_Opacity/masks/Lung_Opacity-518.png,True,False,299,299,L,128.98,b762b7beed34ac4ee286cdb6c872f24f
Lung_Opacity-5180.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5180.png,Lung_Opacity/images/Lung_Opacity-5180.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5180.png,Lung_Opacity/masks/Lung_Opacity-5180.png,True,False,299,299,L,127.87,1fd6d6c65ac68082e064be8572f4f0a4
Lung_Opacity-5181.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5181.png,Lung_Opacity/images/Lung_Opacity-5181.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5181.png,Lung_Opacity/masks/Lung_Opacity-5181.png,True,False,299,299,L,101.19,ef8a1b6c9f51fbef7f36aa46a76be764
Lung_Opacity-5182.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5182.png,Lung_Opacity/images/Lung_Opacity-5182.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5182.png,Lung_Opacity/masks/Lung_Opacity-5182.png,True,False,299,299,L,112.5,0ae852e1c567a50f2a10569669c1ef54
Lung_Opacity-5183.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5183.png,Lung_Opacity/images/Lung_Opacity-5183.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5183.png,Lung_Opacity/masks/Lung_Opacity-5183.png,True,False,299,299,L,146.39,1d7e4308579639a7e942076e7f262146
Lung_Opacity-5184.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5184.png,Lung_Opacity/images/Lung_Opacity-5184.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5184.png,Lung_Opacity/masks/Lung_Opacity-5184.png,True,False,299,299,L,137.05,fdcf902385f30a4660d77e19e53310f2
Lung_Opacity-5185.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5185.png,Lung_Opacity/images/Lung_Opacity-5185.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5185.png,Lung_Opacity/masks/Lung_Opacity-5185.png,True,False,299,299,L,111.94,92d5662767bebea4181ada9bf16fd5a5
Lung_Opacity-5186.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5186.png,Lung_Opacity/images/Lung_Opacity-5186.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5186.png,Lung_Opacity/masks/Lung_Opacity-5186.png,True,False,299,299,L,125.07,27f2d8b58326d6e7ead50f414367c36a
Lung_Opacity-5187.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5187.png,Lung_Opacity/images/Lung_Opacity-5187.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5187.png,Lung_Opacity/masks/Lung_Opacity-5187.png,True,False,299,299,L,167.61,9f323083b11ba9f0e3497ada8ec6c73c
Lung_Opacity-5188.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5188.png,Lung_Opacity/images/Lung_Opacity-5188.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5188.png,Lung_Opacity/masks/Lung_Opacity-5188.png,True,False,299,299,L,151.53,f670f43d3bf279e039d26c0aa49f8aed
Lung_Opacity-5189.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5189.png,Lung_Opacity/images/Lung_Opacity-5189.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5189.png,Lung_Opacity/masks/Lung_Opacity-5189.png,True,False,299,299,L,160.5,ef481dec30be5dbe6a74f9a36a2220b5
Lung_Opacity-519.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-519.png,Lung_Opacity/images/Lung_Opacity-519.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-519.png,Lung_Opacity/masks/Lung_Opacity-519.png,True,False,299,299,L,115.64,ed3a9261c7ee5714bab1467e5eb91451
Lung_Opacity-5190.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5190.png,Lung_Opacity/images/Lung_Opacity-5190.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5190.png,Lung_Opacity/masks/Lung_Opacity-5190.png,True,False,299,299,L,96.62,a42d8fd1eb5771c985c25a3cb7f7a367
Lung_Opacity-5191.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5191.png,Lung_Opacity/images/Lung_Opacity-5191.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5191.png,Lung_Opacity/masks/Lung_Opacity-5191.png,True,False,299,299,L,113.47,0a8861b2e8385e63b2718751fb08004a
Lung_Opacity-5192.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5192.png,Lung_Opacity/images/Lung_Opacity-5192.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5192.png,Lung_Opacity/masks/Lung_Opacity-5192.png,True,False,299,299,L,160.77,10894b956aa17ed4398782aae8bdec2c
Lung_Opacity-5193.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5193.png,Lung_Opacity/images/Lung_Opacity-5193.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5193.png,Lung_Opacity/masks/Lung_Opacity-5193.png,True,False,299,299,L,115.25,9df6825c9353fede206eb2afacf291ea
Lung_Opacity-5194.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5194.png,Lung_Opacity/images/Lung_Opacity-5194.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5194.png,Lung_Opacity/masks/Lung_Opacity-5194.png,True,False,299,299,L,98.52,b2c1449faa9bd198751c1e5bf79a7b94
Lung_Opacity-5195.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5195.png,Lung_Opacity/images/Lung_Opacity-5195.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5195.png,Lung_Opacity/masks/Lung_Opacity-5195.png,True,False,299,299,L,110.14,9d1eb486f835ea9fd1619d0477b3438a
Lung_Opacity-5196.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5196.png,Lung_Opacity/images/Lung_Opacity-5196.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5196.png,Lung_Opacity/masks/Lung_Opacity-5196.png,True,False,299,299,L,159.01,c4053a33defc99eb61ab3708eed68fac
Lung_Opacity-5197.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5197.png,Lung_Opacity/images/Lung_Opacity-5197.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5197.png,Lung_Opacity/masks/Lung_Opacity-5197.png,True,False,299,299,L,139.94,8d3a8ac6f33a7d215da9715735e66797
Lung_Opacity-5198.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5198.png,Lung_Opacity/images/Lung_Opacity-5198.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5198.png,Lung_Opacity/masks/Lung_Opacity-5198.png,True,False,299,299,L,160.38,2392493eba5939884189fa69d6cb465a
Lung_Opacity-5199.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5199.png,Lung_Opacity/images/Lung_Opacity-5199.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5199.png,Lung_Opacity/masks/Lung_Opacity-5199.png,True,False,299,299,L,137.65,fa96ed6a515d1114b4f6367c830288fc
Lung_Opacity-52.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-52.png,Lung_Opacity/images/Lung_Opacity-52.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-52.png,Lung_Opacity/masks/Lung_Opacity-52.png,True,False,299,299,L,115.21,dffe2e42535d733f252cd0b851a21090
Lung_Opacity-520.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-520.png,Lung_Opacity/images/Lung_Opacity-520.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-520.png,Lung_Opacity/masks/Lung_Opacity-520.png,True,False,299,299,L,170.69,c7c64326ac6e2b47f6c764f8e7001e82
Lung_Opacity-5200.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5200.png,Lung_Opacity/images/Lung_Opacity-5200.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5200.png,Lung_Opacity/masks/Lung_Opacity-5200.png,True,False,299,299,L,114.95,84b99334ab3626b6e21ef1d15b10de08
Lung_Opacity-5201.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5201.png,Lung_Opacity/images/Lung_Opacity-5201.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5201.png,Lung_Opacity/masks/Lung_Opacity-5201.png,True,False,299,299,L,117.67,aef36e6cdbfc99f39246436861ca2616
Lung_Opacity-5202.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5202.png,Lung_Opacity/images/Lung_Opacity-5202.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5202.png,Lung_Opacity/masks/Lung_Opacity-5202.png,True,False,299,299,L,140.43,e9c69907cc78f755164ed7033e5c018f
Lung_Opacity-5203.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5203.png,Lung_Opacity/images/Lung_Opacity-5203.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5203.png,Lung_Opacity/masks/Lung_Opacity-5203.png,True,False,299,299,L,110.75,e83d67f52c4b12b5491b303503ce0a5a
Lung_Opacity-5204.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5204.png,Lung_Opacity/images/Lung_Opacity-5204.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5204.png,Lung_Opacity/masks/Lung_Opacity-5204.png,True,False,299,299,L,172.44,2518d8ce60243ef30c4fc3bb523fd5ca
Lung_Opacity-5205.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5205.png,Lung_Opacity/images/Lung_Opacity-5205.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5205.png,Lung_Opacity/masks/Lung_Opacity-5205.png,True,False,299,299,L,146.4,bc01f80e0a4ce5b282ca59d91a696aaf
Lung_Opacity-5206.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5206.png,Lung_Opacity/images/Lung_Opacity-5206.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5206.png,Lung_Opacity/masks/Lung_Opacity-5206.png,True,False,299,299,L,108.43,5f3e4dee0b1c4855da9e33010681b7b1
Lung_Opacity-5207.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5207.png,Lung_Opacity/images/Lung_Opacity-5207.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5207.png,Lung_Opacity/masks/Lung_Opacity-5207.png,True,False,299,299,L,86.32,afbfc57ba2503481c850e62e82e53bb7
Lung_Opacity-5208.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5208.png,Lung_Opacity/images/Lung_Opacity-5208.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5208.png,Lung_Opacity/masks/Lung_Opacity-5208.png,True,False,299,299,L,96.44,fab0344123481b1b9dbe7ae427ad7ac7
Lung_Opacity-5209.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5209.png,Lung_Opacity/images/Lung_Opacity-5209.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5209.png,Lung_Opacity/masks/Lung_Opacity-5209.png,True,False,299,299,L,141.92,7abe4c73a055a865e4f2963ed2bd4e80
Lung_Opacity-521.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-521.png,Lung_Opacity/images/Lung_Opacity-521.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-521.png,Lung_Opacity/masks/Lung_Opacity-521.png,True,False,299,299,L,159.21,0ca8d4c5d408275d2d15ccdc622fbcf4
Lung_Opacity-5210.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5210.png,Lung_Opacity/images/Lung_Opacity-5210.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5210.png,Lung_Opacity/masks/Lung_Opacity-5210.png,True,False,299,299,L,145.61,eb78175784f8fa58f7355fa98cbc1ba9
Lung_Opacity-5211.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5211.png,Lung_Opacity/images/Lung_Opacity-5211.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5211.png,Lung_Opacity/masks/Lung_Opacity-5211.png,True,False,299,299,L,109.16,3016ac571ff6058276d7e78879157db4
Lung_Opacity-5212.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5212.png,Lung_Opacity/images/Lung_Opacity-5212.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5212.png,Lung_Opacity/masks/Lung_Opacity-5212.png,True,False,299,299,L,117.06,1db2b40a06e6e5d0559e7bf7d9db32a1
Lung_Opacity-5213.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5213.png,Lung_Opacity/images/Lung_Opacity-5213.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5213.png,Lung_Opacity/masks/Lung_Opacity-5213.png,True,False,299,299,L,113.47,7a4e47c133c1e7b60abb3275f345a714
Lung_Opacity-5214.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5214.png,Lung_Opacity/images/Lung_Opacity-5214.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5214.png,Lung_Opacity/masks/Lung_Opacity-5214.png,True,False,299,299,L,111.39,5d8f7d3a747294eb6f8a7faf528aa750
Lung_Opacity-5215.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5215.png,Lung_Opacity/images/Lung_Opacity-5215.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5215.png,Lung_Opacity/masks/Lung_Opacity-5215.png,True,False,299,299,L,134.34,e0f1980a0ad69f1a086888e3709b7c1e
Lung_Opacity-5216.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5216.png,Lung_Opacity/images/Lung_Opacity-5216.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5216.png,Lung_Opacity/masks/Lung_Opacity-5216.png,True,False,299,299,L,138.95,72957955cd096f2e61ba7602463037bb
Lung_Opacity-5217.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5217.png,Lung_Opacity/images/Lung_Opacity-5217.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5217.png,Lung_Opacity/masks/Lung_Opacity-5217.png,True,False,299,299,L,114.8,3af86bb604ad9fdc57091bd04b3f1265
Lung_Opacity-5218.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5218.png,Lung_Opacity/images/Lung_Opacity-5218.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5218.png,Lung_Opacity/masks/Lung_Opacity-5218.png,True,False,299,299,L,111.64,d8d41b12599e50e96feefc9b04a9e9dc
Lung_Opacity-5219.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5219.png,Lung_Opacity/images/Lung_Opacity-5219.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5219.png,Lung_Opacity/masks/Lung_Opacity-5219.png,True,False,299,299,L,122.36,ed48d05eb68219b7c459cb98b2245db6
Lung_Opacity-522.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-522.png,Lung_Opacity/images/Lung_Opacity-522.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-522.png,Lung_Opacity/masks/Lung_Opacity-522.png,True,False,299,299,L,158.19,12936bcb4eec8f4646c429847f21c68a
Lung_Opacity-5220.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5220.png,Lung_Opacity/images/Lung_Opacity-5220.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5220.png,Lung_Opacity/masks/Lung_Opacity-5220.png,True,False,299,299,L,151.48,44501e29dd293daa59d8339692636f67
Lung_Opacity-5221.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5221.png,Lung_Opacity/images/Lung_Opacity-5221.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5221.png,Lung_Opacity/masks/Lung_Opacity-5221.png,True,False,299,299,L,128.02,83afb11342143611a3b5edfa120ce929
Lung_Opacity-5222.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5222.png,Lung_Opacity/images/Lung_Opacity-5222.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5222.png,Lung_Opacity/masks/Lung_Opacity-5222.png,True,False,299,299,L,117.14,19fcd645f3d303a438eb3b1228d69261
Lung_Opacity-5223.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5223.png,Lung_Opacity/images/Lung_Opacity-5223.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5223.png,Lung_Opacity/masks/Lung_Opacity-5223.png,True,False,299,299,L,106.85,a6682d4949b043da9b94f15f9fedb3e9
Lung_Opacity-5224.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5224.png,Lung_Opacity/images/Lung_Opacity-5224.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5224.png,Lung_Opacity/masks/Lung_Opacity-5224.png,True,False,299,299,L,101.54,183d8d7174f05b547b001c44b2b4290b
Lung_Opacity-5225.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5225.png,Lung_Opacity/images/Lung_Opacity-5225.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5225.png,Lung_Opacity/masks/Lung_Opacity-5225.png,True,False,299,299,L,142.18,1164ea8bdd291114268a676382058ea0
Lung_Opacity-5226.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5226.png,Lung_Opacity/images/Lung_Opacity-5226.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5226.png,Lung_Opacity/masks/Lung_Opacity-5226.png,True,False,299,299,L,138.71,25ea6a0fece425ecec6da3ffe781860f
Lung_Opacity-5227.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5227.png,Lung_Opacity/images/Lung_Opacity-5227.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5227.png,Lung_Opacity/masks/Lung_Opacity-5227.png,True,False,299,299,L,104.18,41aa530a5ead04790926110e0cdf078e
Lung_Opacity-5228.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5228.png,Lung_Opacity/images/Lung_Opacity-5228.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5228.png,Lung_Opacity/masks/Lung_Opacity-5228.png,True,False,299,299,L,104.49,9c1b1ab40039846bf1ab766c4a6472b9
Lung_Opacity-5229.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5229.png,Lung_Opacity/images/Lung_Opacity-5229.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5229.png,Lung_Opacity/masks/Lung_Opacity-5229.png,True,False,299,299,L,113.84,2aa81ca53c1be64fc1301402f05fa009
Lung_Opacity-523.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-523.png,Lung_Opacity/images/Lung_Opacity-523.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-523.png,Lung_Opacity/masks/Lung_Opacity-523.png,True,False,299,299,L,160.18,c83b747a3fb0b14cdeb7fa51dd298453
Lung_Opacity-5230.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5230.png,Lung_Opacity/images/Lung_Opacity-5230.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5230.png,Lung_Opacity/masks/Lung_Opacity-5230.png,True,False,299,299,L,154.49,861ca6b3bb20ff716cd09edbef681517
Lung_Opacity-5231.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5231.png,Lung_Opacity/images/Lung_Opacity-5231.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5231.png,Lung_Opacity/masks/Lung_Opacity-5231.png,True,False,299,299,L,120.66,0b4ac2dc40bab542be8eca94db0073ff
Lung_Opacity-5232.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5232.png,Lung_Opacity/images/Lung_Opacity-5232.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5232.png,Lung_Opacity/masks/Lung_Opacity-5232.png,True,False,299,299,L,142.66,d0aada529a969849214314a1ef8b04ff
Lung_Opacity-5233.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5233.png,Lung_Opacity/images/Lung_Opacity-5233.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5233.png,Lung_Opacity/masks/Lung_Opacity-5233.png,True,False,299,299,L,68.03,505fa1f76ff0190806fbd6c42f86970f
Lung_Opacity-5234.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5234.png,Lung_Opacity/images/Lung_Opacity-5234.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5234.png,Lung_Opacity/masks/Lung_Opacity-5234.png,True,False,299,299,L,142.93,1b64f6b39acc874c124d3cae895eb2d2
Lung_Opacity-5235.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5235.png,Lung_Opacity/images/Lung_Opacity-5235.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5235.png,Lung_Opacity/masks/Lung_Opacity-5235.png,True,False,299,299,L,108.07,33ea18a7fc3845e24e4037dcbd7f85f2
Lung_Opacity-5236.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5236.png,Lung_Opacity/images/Lung_Opacity-5236.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5236.png,Lung_Opacity/masks/Lung_Opacity-5236.png,True,False,299,299,L,175.34,bdfc9738ea9c4c5d7e3972822b33251b
Lung_Opacity-5237.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5237.png,Lung_Opacity/images/Lung_Opacity-5237.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5237.png,Lung_Opacity/masks/Lung_Opacity-5237.png,True,False,299,299,L,173.42,8f7f987beef80b07a980faca18c9f9d3
Lung_Opacity-5238.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5238.png,Lung_Opacity/images/Lung_Opacity-5238.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5238.png,Lung_Opacity/masks/Lung_Opacity-5238.png,True,False,299,299,L,113.35,e924c73eefe6a846f42910ff9331d85f
Lung_Opacity-5239.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5239.png,Lung_Opacity/images/Lung_Opacity-5239.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5239.png,Lung_Opacity/masks/Lung_Opacity-5239.png,True,False,299,299,L,169.35,c17603713a9b2a6adfb2533bde636368
Lung_Opacity-524.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-524.png,Lung_Opacity/images/Lung_Opacity-524.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-524.png,Lung_Opacity/masks/Lung_Opacity-524.png,True,False,299,299,L,53.36,3b6250ec257ab637b57781e0f4aeaa11
Lung_Opacity-5240.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5240.png,Lung_Opacity/images/Lung_Opacity-5240.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5240.png,Lung_Opacity/masks/Lung_Opacity-5240.png,True,False,299,299,L,124.0,2eba0694a18ff694126fdbb0d3920731
Lung_Opacity-5241.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5241.png,Lung_Opacity/images/Lung_Opacity-5241.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5241.png,Lung_Opacity/masks/Lung_Opacity-5241.png,True,False,299,299,L,128.22,daf726d7374380d19d9384dc2b22c823
Lung_Opacity-5242.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5242.png,Lung_Opacity/images/Lung_Opacity-5242.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5242.png,Lung_Opacity/masks/Lung_Opacity-5242.png,True,False,299,299,L,132.24,cdde5bce662574182b6ce244d02f9fe7
Lung_Opacity-5243.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5243.png,Lung_Opacity/images/Lung_Opacity-5243.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5243.png,Lung_Opacity/masks/Lung_Opacity-5243.png,True,False,299,299,L,106.66,b356d047880449970bf2b820563213cd
Lung_Opacity-5244.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5244.png,Lung_Opacity/images/Lung_Opacity-5244.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5244.png,Lung_Opacity/masks/Lung_Opacity-5244.png,True,False,299,299,L,157.36,6c4adfa1e6479669273accf136d4aef8
Lung_Opacity-5245.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5245.png,Lung_Opacity/images/Lung_Opacity-5245.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5245.png,Lung_Opacity/masks/Lung_Opacity-5245.png,True,False,299,299,L,115.96,c09a0a03d7c1d93e6c541a20d80c7435
Lung_Opacity-5246.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5246.png,Lung_Opacity/images/Lung_Opacity-5246.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5246.png,Lung_Opacity/masks/Lung_Opacity-5246.png,True,False,299,299,L,121.76,e63f6e456cb29862a35cb8dfe3bc5063
Lung_Opacity-5247.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5247.png,Lung_Opacity/images/Lung_Opacity-5247.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5247.png,Lung_Opacity/masks/Lung_Opacity-5247.png,True,False,299,299,L,167.95,304c15b11c30d5773d6a59ddf9c2e84f
Lung_Opacity-5248.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5248.png,Lung_Opacity/images/Lung_Opacity-5248.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5248.png,Lung_Opacity/masks/Lung_Opacity-5248.png,True,False,299,299,L,107.27,2f7fa7bb2cd7ad572e2c7971ff35c4fe
Lung_Opacity-5249.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5249.png,Lung_Opacity/images/Lung_Opacity-5249.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5249.png,Lung_Opacity/masks/Lung_Opacity-5249.png,True,False,299,299,L,126.28,bd0dcec797e4640cf645132c55e76f88
Lung_Opacity-525.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-525.png,Lung_Opacity/images/Lung_Opacity-525.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-525.png,Lung_Opacity/masks/Lung_Opacity-525.png,True,False,299,299,L,92.41,55400bd34d5597b6344c836fdf8b1317
Lung_Opacity-5250.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5250.png,Lung_Opacity/images/Lung_Opacity-5250.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5250.png,Lung_Opacity/masks/Lung_Opacity-5250.png,True,False,299,299,L,116.02,4b5cd7f40d78ad2aec76c427b8667b85
Lung_Opacity-5251.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5251.png,Lung_Opacity/images/Lung_Opacity-5251.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5251.png,Lung_Opacity/masks/Lung_Opacity-5251.png,True,False,299,299,L,114.48,b767e512e7c99db63cbb2a26b759fcae
Lung_Opacity-5252.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5252.png,Lung_Opacity/images/Lung_Opacity-5252.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5252.png,Lung_Opacity/masks/Lung_Opacity-5252.png,True,False,299,299,L,155.33,b581ae3789504f89220353333f3aa8ea
Lung_Opacity-5253.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5253.png,Lung_Opacity/images/Lung_Opacity-5253.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5253.png,Lung_Opacity/masks/Lung_Opacity-5253.png,True,False,299,299,L,106.44,2bf82ccc9e667629addecb63bf45c78c
Lung_Opacity-5254.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5254.png,Lung_Opacity/images/Lung_Opacity-5254.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5254.png,Lung_Opacity/masks/Lung_Opacity-5254.png,True,False,299,299,L,155.12,1582267af72d7c74758449edd3374805
Lung_Opacity-5255.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5255.png,Lung_Opacity/images/Lung_Opacity-5255.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5255.png,Lung_Opacity/masks/Lung_Opacity-5255.png,True,False,299,299,L,120.87,afbcc0f6463d07daca458332edab7bc3
Lung_Opacity-5256.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5256.png,Lung_Opacity/images/Lung_Opacity-5256.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5256.png,Lung_Opacity/masks/Lung_Opacity-5256.png,True,False,299,299,L,91.19,ce4f6e813c33061b0a3f22fd00925b05
Lung_Opacity-5257.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5257.png,Lung_Opacity/images/Lung_Opacity-5257.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5257.png,Lung_Opacity/masks/Lung_Opacity-5257.png,True,False,299,299,L,121.33,f62eeae5888cb93dbe13e6f1da4bd776
Lung_Opacity-5258.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5258.png,Lung_Opacity/images/Lung_Opacity-5258.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5258.png,Lung_Opacity/masks/Lung_Opacity-5258.png,True,False,299,299,L,146.99,4d66a23323624bf18e25c0d1c8aed9d0
Lung_Opacity-5259.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5259.png,Lung_Opacity/images/Lung_Opacity-5259.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5259.png,Lung_Opacity/masks/Lung_Opacity-5259.png,True,False,299,299,L,125.96,16bef9c71e64ca053a018387e6382a33
Lung_Opacity-526.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-526.png,Lung_Opacity/images/Lung_Opacity-526.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-526.png,Lung_Opacity/masks/Lung_Opacity-526.png,True,False,299,299,L,173.47,815787a83e82985e7717872bec0fc240
Lung_Opacity-5260.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5260.png,Lung_Opacity/images/Lung_Opacity-5260.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5260.png,Lung_Opacity/masks/Lung_Opacity-5260.png,True,False,299,299,L,117.75,8d6681a99e5d145e625366d274624bf0
Lung_Opacity-5261.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5261.png,Lung_Opacity/images/Lung_Opacity-5261.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5261.png,Lung_Opacity/masks/Lung_Opacity-5261.png,True,False,299,299,L,104.3,7f2a30c593f86de786a99ac2e9294cee
Lung_Opacity-5262.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5262.png,Lung_Opacity/images/Lung_Opacity-5262.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5262.png,Lung_Opacity/masks/Lung_Opacity-5262.png,True,False,299,299,L,125.1,d567ebf52904e97a382d58f4d30a04d4
Lung_Opacity-5263.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5263.png,Lung_Opacity/images/Lung_Opacity-5263.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5263.png,Lung_Opacity/masks/Lung_Opacity-5263.png,True,False,299,299,L,133.14,fdf0daa3b292edba080b6e3ec410b8cf
Lung_Opacity-5264.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5264.png,Lung_Opacity/images/Lung_Opacity-5264.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5264.png,Lung_Opacity/masks/Lung_Opacity-5264.png,True,False,299,299,L,122.14,c6f38c4db28020d4a16483ba55dd1cf3
Lung_Opacity-5265.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5265.png,Lung_Opacity/images/Lung_Opacity-5265.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5265.png,Lung_Opacity/masks/Lung_Opacity-5265.png,True,False,299,299,L,129.94,93558debb114477a558fd36945a4601c
Lung_Opacity-5266.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5266.png,Lung_Opacity/images/Lung_Opacity-5266.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5266.png,Lung_Opacity/masks/Lung_Opacity-5266.png,True,False,299,299,L,119.98,6fc872fb21a8e9429250569aa1057ed7
Lung_Opacity-5267.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5267.png,Lung_Opacity/images/Lung_Opacity-5267.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5267.png,Lung_Opacity/masks/Lung_Opacity-5267.png,True,False,299,299,L,138.44,15c368c61d934983086d661af11542cb
Lung_Opacity-5268.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5268.png,Lung_Opacity/images/Lung_Opacity-5268.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5268.png,Lung_Opacity/masks/Lung_Opacity-5268.png,True,False,299,299,L,166.81,401ab787a5ab251bb7d3b2d871bd78fc
Lung_Opacity-5269.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5269.png,Lung_Opacity/images/Lung_Opacity-5269.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5269.png,Lung_Opacity/masks/Lung_Opacity-5269.png,True,False,299,299,L,102.49,5b6fd20d940b74b31d062797b9449923
Lung_Opacity-527.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-527.png,Lung_Opacity/images/Lung_Opacity-527.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-527.png,Lung_Opacity/masks/Lung_Opacity-527.png,True,False,299,299,L,109.4,b26a089955e06e3a53c52e8013423303
Lung_Opacity-5270.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5270.png,Lung_Opacity/images/Lung_Opacity-5270.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5270.png,Lung_Opacity/masks/Lung_Opacity-5270.png,True,False,299,299,L,109.39,9d33fbc477683e36b96041bf3f5edc04
Lung_Opacity-5271.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5271.png,Lung_Opacity/images/Lung_Opacity-5271.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5271.png,Lung_Opacity/masks/Lung_Opacity-5271.png,True,False,299,299,L,127.04,b7d171593264fde1adcf47d617d9d3e7
Lung_Opacity-5272.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5272.png,Lung_Opacity/images/Lung_Opacity-5272.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5272.png,Lung_Opacity/masks/Lung_Opacity-5272.png,True,False,299,299,L,115.52,c04b00a7568cd1af4f75b2f3a6e8171a
Lung_Opacity-5273.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5273.png,Lung_Opacity/images/Lung_Opacity-5273.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5273.png,Lung_Opacity/masks/Lung_Opacity-5273.png,True,False,299,299,L,117.03,d743500ad3bf07f993717caaf23b0882
Lung_Opacity-5274.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5274.png,Lung_Opacity/images/Lung_Opacity-5274.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5274.png,Lung_Opacity/masks/Lung_Opacity-5274.png,True,False,299,299,L,121.72,2c333616e4be7e84269c72470ebba2c1
Lung_Opacity-5275.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5275.png,Lung_Opacity/images/Lung_Opacity-5275.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5275.png,Lung_Opacity/masks/Lung_Opacity-5275.png,True,False,299,299,L,106.88,484f0aaec338c2da4e2d1106ad07be82
Lung_Opacity-5276.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5276.png,Lung_Opacity/images/Lung_Opacity-5276.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5276.png,Lung_Opacity/masks/Lung_Opacity-5276.png,True,False,299,299,L,114.68,39234e40f770f8ff311b80004eb7eb58
Lung_Opacity-5277.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5277.png,Lung_Opacity/images/Lung_Opacity-5277.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5277.png,Lung_Opacity/masks/Lung_Opacity-5277.png,True,False,299,299,L,151.06,c5dd7526ad79e09898bd95eb171b479b
Lung_Opacity-5278.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5278.png,Lung_Opacity/images/Lung_Opacity-5278.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5278.png,Lung_Opacity/masks/Lung_Opacity-5278.png,True,False,299,299,L,136.44,ba980d2f3e88200998d8cf32549a847a
Lung_Opacity-5279.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5279.png,Lung_Opacity/images/Lung_Opacity-5279.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5279.png,Lung_Opacity/masks/Lung_Opacity-5279.png,True,False,299,299,L,126.12,6434f1242cf2fede510148a734208155
Lung_Opacity-528.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-528.png,Lung_Opacity/images/Lung_Opacity-528.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-528.png,Lung_Opacity/masks/Lung_Opacity-528.png,True,False,299,299,L,157.31,75f94061b859297bd067df93eacb9a9d
Lung_Opacity-5280.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5280.png,Lung_Opacity/images/Lung_Opacity-5280.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5280.png,Lung_Opacity/masks/Lung_Opacity-5280.png,True,False,299,299,L,165.92,6c4274f93bba795fa942a4d863358d1c
Lung_Opacity-5281.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5281.png,Lung_Opacity/images/Lung_Opacity-5281.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5281.png,Lung_Opacity/masks/Lung_Opacity-5281.png,True,False,299,299,L,111.93,109c61cbfefd3285690c70a68e52ed70
Lung_Opacity-5282.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5282.png,Lung_Opacity/images/Lung_Opacity-5282.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5282.png,Lung_Opacity/masks/Lung_Opacity-5282.png,True,False,299,299,L,73.47,0bc396a45be6ac039362f95872f1468d
Lung_Opacity-5283.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5283.png,Lung_Opacity/images/Lung_Opacity-5283.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5283.png,Lung_Opacity/masks/Lung_Opacity-5283.png,True,False,299,299,L,118.65,d7b0f9a9aa01f1a36cd576c5bb268327
Lung_Opacity-5284.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5284.png,Lung_Opacity/images/Lung_Opacity-5284.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5284.png,Lung_Opacity/masks/Lung_Opacity-5284.png,True,False,299,299,L,129.62,8232b7e8202b62e80abd91c3802a1e17
Lung_Opacity-5285.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5285.png,Lung_Opacity/images/Lung_Opacity-5285.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5285.png,Lung_Opacity/masks/Lung_Opacity-5285.png,True,False,299,299,L,149.74,b8e2cfe4ceb1a90fa8e09eca48252595
Lung_Opacity-5286.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5286.png,Lung_Opacity/images/Lung_Opacity-5286.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5286.png,Lung_Opacity/masks/Lung_Opacity-5286.png,True,False,299,299,L,116.03,23dba10f0d9e861972492316f2df313e
Lung_Opacity-5287.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5287.png,Lung_Opacity/images/Lung_Opacity-5287.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5287.png,Lung_Opacity/masks/Lung_Opacity-5287.png,True,False,299,299,L,111.11,6bd4f74bdab2ddebdf69a5c4f71ee25d
Lung_Opacity-5288.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5288.png,Lung_Opacity/images/Lung_Opacity-5288.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5288.png,Lung_Opacity/masks/Lung_Opacity-5288.png,True,False,299,299,L,121.01,ebaab069df32686b09eb230f5a49179d
Lung_Opacity-5289.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5289.png,Lung_Opacity/images/Lung_Opacity-5289.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5289.png,Lung_Opacity/masks/Lung_Opacity-5289.png,True,False,299,299,L,111.38,34ade64823da2702b5321dbbfdacf5c6
Lung_Opacity-529.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-529.png,Lung_Opacity/images/Lung_Opacity-529.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-529.png,Lung_Opacity/masks/Lung_Opacity-529.png,True,False,299,299,L,111.88,58f3e660901a8e850d6dea8a7dc23fac
Lung_Opacity-5290.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5290.png,Lung_Opacity/images/Lung_Opacity-5290.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5290.png,Lung_Opacity/masks/Lung_Opacity-5290.png,True,False,299,299,L,110.98,b70407b72821eb20cf52b86f05ab79a8
Lung_Opacity-5291.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5291.png,Lung_Opacity/images/Lung_Opacity-5291.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5291.png,Lung_Opacity/masks/Lung_Opacity-5291.png,True,False,299,299,L,157.17,a683f8b04ddfbe0c0113ce8f80cdd913
Lung_Opacity-5292.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5292.png,Lung_Opacity/images/Lung_Opacity-5292.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5292.png,Lung_Opacity/masks/Lung_Opacity-5292.png,True,False,299,299,L,139.6,af8cd1020d52c392737227aba63d9baf
Lung_Opacity-5293.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5293.png,Lung_Opacity/images/Lung_Opacity-5293.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5293.png,Lung_Opacity/masks/Lung_Opacity-5293.png,True,False,299,299,L,106.35,68501c58dbc4632cbf90d2c2037c8a04
Lung_Opacity-5294.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5294.png,Lung_Opacity/images/Lung_Opacity-5294.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5294.png,Lung_Opacity/masks/Lung_Opacity-5294.png,True,False,299,299,L,105.68,95b058109532b097513363be01a14a07
Lung_Opacity-5295.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5295.png,Lung_Opacity/images/Lung_Opacity-5295.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5295.png,Lung_Opacity/masks/Lung_Opacity-5295.png,True,False,299,299,L,122.42,18a157860e37770cb20be0adbbbd3399
Lung_Opacity-5296.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5296.png,Lung_Opacity/images/Lung_Opacity-5296.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5296.png,Lung_Opacity/masks/Lung_Opacity-5296.png,True,False,299,299,L,94.79,6f2d7b89adcdffa15f2ced80de8c8e79
Lung_Opacity-5297.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5297.png,Lung_Opacity/images/Lung_Opacity-5297.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5297.png,Lung_Opacity/masks/Lung_Opacity-5297.png,True,False,299,299,L,132.38,a89a71e8b1153e9ffcb0ba80edea0c5e
Lung_Opacity-5298.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5298.png,Lung_Opacity/images/Lung_Opacity-5298.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5298.png,Lung_Opacity/masks/Lung_Opacity-5298.png,True,False,299,299,L,112.38,c24bc55cbb6dd149d67f35ca9013d0f1
Lung_Opacity-5299.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5299.png,Lung_Opacity/images/Lung_Opacity-5299.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5299.png,Lung_Opacity/masks/Lung_Opacity-5299.png,True,False,299,299,L,104.41,e54643f9b0607e0313d9b96011ac199d
Lung_Opacity-53.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-53.png,Lung_Opacity/images/Lung_Opacity-53.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-53.png,Lung_Opacity/masks/Lung_Opacity-53.png,True,False,299,299,L,118.44,8c0b7382c03b5dfe4062e12cbcc62871
Lung_Opacity-530.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-530.png,Lung_Opacity/images/Lung_Opacity-530.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-530.png,Lung_Opacity/masks/Lung_Opacity-530.png,True,False,299,299,L,174.99,9e478a7098d430d7b4bace71bc7b0cb4
Lung_Opacity-5300.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5300.png,Lung_Opacity/images/Lung_Opacity-5300.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5300.png,Lung_Opacity/masks/Lung_Opacity-5300.png,True,False,299,299,L,135.0,82b1556e7c63f5e75fcaf5a8d0a02a5a
Lung_Opacity-5301.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5301.png,Lung_Opacity/images/Lung_Opacity-5301.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5301.png,Lung_Opacity/masks/Lung_Opacity-5301.png,True,False,299,299,L,147.45,a09cd874cd5d6214025b2b5c9a27dc94
Lung_Opacity-5302.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5302.png,Lung_Opacity/images/Lung_Opacity-5302.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5302.png,Lung_Opacity/masks/Lung_Opacity-5302.png,True,False,299,299,L,119.41,9eb95014e65cb9595f3dfe2a2b8f6821
Lung_Opacity-5303.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5303.png,Lung_Opacity/images/Lung_Opacity-5303.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5303.png,Lung_Opacity/masks/Lung_Opacity-5303.png,True,False,299,299,L,173.46,b0a6d7e52ad353dff431781f09f078e7
Lung_Opacity-5304.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5304.png,Lung_Opacity/images/Lung_Opacity-5304.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5304.png,Lung_Opacity/masks/Lung_Opacity-5304.png,True,False,299,299,L,149.35,553a6e775d17c659fedd205df435d985
Lung_Opacity-5305.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5305.png,Lung_Opacity/images/Lung_Opacity-5305.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5305.png,Lung_Opacity/masks/Lung_Opacity-5305.png,True,False,299,299,L,93.69,6a9b520e4374b2bec749b2e95eebd19d
Lung_Opacity-5306.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5306.png,Lung_Opacity/images/Lung_Opacity-5306.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5306.png,Lung_Opacity/masks/Lung_Opacity-5306.png,True,False,299,299,L,118.3,ed8e7298d69eca8a25b133942eba25e9
Lung_Opacity-5307.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5307.png,Lung_Opacity/images/Lung_Opacity-5307.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5307.png,Lung_Opacity/masks/Lung_Opacity-5307.png,True,False,299,299,L,109.35,43e8dade12510c58d35a23a8fef24f30
Lung_Opacity-5308.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5308.png,Lung_Opacity/images/Lung_Opacity-5308.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5308.png,Lung_Opacity/masks/Lung_Opacity-5308.png,True,False,299,299,L,111.23,54b290544610d6591d4e79396685d999
Lung_Opacity-5309.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5309.png,Lung_Opacity/images/Lung_Opacity-5309.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5309.png,Lung_Opacity/masks/Lung_Opacity-5309.png,True,False,299,299,L,105.06,e0465d8dec0f44f6b0c226a921d56008
Lung_Opacity-531.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-531.png,Lung_Opacity/images/Lung_Opacity-531.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-531.png,Lung_Opacity/masks/Lung_Opacity-531.png,True,False,299,299,L,160.48,7a125ca38452764b5726091a9e66229d
Lung_Opacity-5310.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5310.png,Lung_Opacity/images/Lung_Opacity-5310.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5310.png,Lung_Opacity/masks/Lung_Opacity-5310.png,True,False,299,299,L,137.86,04a4af656e0e34957953d72f47f81b08
Lung_Opacity-5311.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5311.png,Lung_Opacity/images/Lung_Opacity-5311.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5311.png,Lung_Opacity/masks/Lung_Opacity-5311.png,True,False,299,299,L,120.29,ddc6fd9f548f56e31573c07d56b3fc70
Lung_Opacity-5312.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5312.png,Lung_Opacity/images/Lung_Opacity-5312.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5312.png,Lung_Opacity/masks/Lung_Opacity-5312.png,True,False,299,299,L,92.2,a807d6e36552d9002c9f1466fea02c92
Lung_Opacity-5313.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5313.png,Lung_Opacity/images/Lung_Opacity-5313.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5313.png,Lung_Opacity/masks/Lung_Opacity-5313.png,True,False,299,299,L,153.21,1e77ef0c4ef72c613dd2918f3372fd9b
Lung_Opacity-5314.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5314.png,Lung_Opacity/images/Lung_Opacity-5314.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5314.png,Lung_Opacity/masks/Lung_Opacity-5314.png,True,False,299,299,L,134.33,0308455fb5d31350f00c94831e605a95
Lung_Opacity-5315.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5315.png,Lung_Opacity/images/Lung_Opacity-5315.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5315.png,Lung_Opacity/masks/Lung_Opacity-5315.png,True,False,299,299,L,137.19,61ea8c604d6eb1a4802b325ec2b33a11
Lung_Opacity-5316.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5316.png,Lung_Opacity/images/Lung_Opacity-5316.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5316.png,Lung_Opacity/masks/Lung_Opacity-5316.png,True,False,299,299,L,155.94,c31e7733ea10fc88b550ed3fe7b49512
Lung_Opacity-5317.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5317.png,Lung_Opacity/images/Lung_Opacity-5317.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5317.png,Lung_Opacity/masks/Lung_Opacity-5317.png,True,False,299,299,L,161.64,411c48dc78d7718a51645b4ecf75369b
Lung_Opacity-5318.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5318.png,Lung_Opacity/images/Lung_Opacity-5318.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5318.png,Lung_Opacity/masks/Lung_Opacity-5318.png,True,False,299,299,L,83.98,8f3f780d268e9f0d11b2a892312ac1a9
Lung_Opacity-5319.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5319.png,Lung_Opacity/images/Lung_Opacity-5319.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5319.png,Lung_Opacity/masks/Lung_Opacity-5319.png,True,False,299,299,L,179.94,62272b8445d712a9ff1cd8d746e119f9
Lung_Opacity-532.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-532.png,Lung_Opacity/images/Lung_Opacity-532.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-532.png,Lung_Opacity/masks/Lung_Opacity-532.png,True,False,299,299,L,107.82,fb95fb182b4925df4f92e4577b1ed9e6
Lung_Opacity-5320.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5320.png,Lung_Opacity/images/Lung_Opacity-5320.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5320.png,Lung_Opacity/masks/Lung_Opacity-5320.png,True,False,299,299,L,119.74,830ed14037998ff57307d21a99449d7e
Lung_Opacity-5321.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5321.png,Lung_Opacity/images/Lung_Opacity-5321.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5321.png,Lung_Opacity/masks/Lung_Opacity-5321.png,True,False,299,299,L,117.68,d4a172225d85a7db8f34ed343860608c
Lung_Opacity-5322.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5322.png,Lung_Opacity/images/Lung_Opacity-5322.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5322.png,Lung_Opacity/masks/Lung_Opacity-5322.png,True,False,299,299,L,120.37,ce7f2112031b23d223da0459456e2256
Lung_Opacity-5323.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5323.png,Lung_Opacity/images/Lung_Opacity-5323.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5323.png,Lung_Opacity/masks/Lung_Opacity-5323.png,True,False,299,299,L,131.48,51dfcd2fcd601fd1018a993a97e8c091
Lung_Opacity-5324.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5324.png,Lung_Opacity/images/Lung_Opacity-5324.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5324.png,Lung_Opacity/masks/Lung_Opacity-5324.png,True,False,299,299,L,144.72,f54fdfe72648816a7cace6fea1b05a57
Lung_Opacity-5325.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5325.png,Lung_Opacity/images/Lung_Opacity-5325.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5325.png,Lung_Opacity/masks/Lung_Opacity-5325.png,True,False,299,299,L,111.96,f1a6edc1b9b88e38759cf0de48e3c7f3
Lung_Opacity-5326.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5326.png,Lung_Opacity/images/Lung_Opacity-5326.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5326.png,Lung_Opacity/masks/Lung_Opacity-5326.png,True,False,299,299,L,134.71,7cb3864400bf58e7048446990d74ef90
Lung_Opacity-5327.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5327.png,Lung_Opacity/images/Lung_Opacity-5327.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5327.png,Lung_Opacity/masks/Lung_Opacity-5327.png,True,False,299,299,L,150.36,e6143aec802499ba37fdeb70620a0962
Lung_Opacity-5328.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5328.png,Lung_Opacity/images/Lung_Opacity-5328.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5328.png,Lung_Opacity/masks/Lung_Opacity-5328.png,True,False,299,299,L,122.27,5d1bcb7919db4d82ee0b379e31840058
Lung_Opacity-5329.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5329.png,Lung_Opacity/images/Lung_Opacity-5329.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5329.png,Lung_Opacity/masks/Lung_Opacity-5329.png,True,False,299,299,L,88.28,6806df794925327cf28d33d6fbbc0c7e
Lung_Opacity-533.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-533.png,Lung_Opacity/images/Lung_Opacity-533.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-533.png,Lung_Opacity/masks/Lung_Opacity-533.png,True,False,299,299,L,97.44,c8b2df1aff83c713022bd44403d7dc63
Lung_Opacity-5330.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5330.png,Lung_Opacity/images/Lung_Opacity-5330.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5330.png,Lung_Opacity/masks/Lung_Opacity-5330.png,True,False,299,299,L,157.25,9ef01c1e73eee09c42e751c6ec754a30
Lung_Opacity-5331.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5331.png,Lung_Opacity/images/Lung_Opacity-5331.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5331.png,Lung_Opacity/masks/Lung_Opacity-5331.png,True,False,299,299,L,142.43,c1ae20856898c1b2f0f53b180434c286
Lung_Opacity-5332.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5332.png,Lung_Opacity/images/Lung_Opacity-5332.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5332.png,Lung_Opacity/masks/Lung_Opacity-5332.png,True,False,299,299,L,113.52,6765394ace9b721250004cbfb72c1663
Lung_Opacity-5333.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5333.png,Lung_Opacity/images/Lung_Opacity-5333.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5333.png,Lung_Opacity/masks/Lung_Opacity-5333.png,True,False,299,299,L,115.07,f55735b7b7c6674717da0940a7bfdfc2
Lung_Opacity-5334.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5334.png,Lung_Opacity/images/Lung_Opacity-5334.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5334.png,Lung_Opacity/masks/Lung_Opacity-5334.png,True,False,299,299,L,118.5,69b1ea0edd1b6e1a5ac52db0a0b9fd4a
Lung_Opacity-5335.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5335.png,Lung_Opacity/images/Lung_Opacity-5335.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5335.png,Lung_Opacity/masks/Lung_Opacity-5335.png,True,False,299,299,L,134.61,873cddc0685099f4def21f7e0911a609
Lung_Opacity-5336.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5336.png,Lung_Opacity/images/Lung_Opacity-5336.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5336.png,Lung_Opacity/masks/Lung_Opacity-5336.png,True,False,299,299,L,142.45,b9ad9261b645d3d64263eda295d0ad26
Lung_Opacity-5337.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5337.png,Lung_Opacity/images/Lung_Opacity-5337.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5337.png,Lung_Opacity/masks/Lung_Opacity-5337.png,True,False,299,299,L,112.8,b7fa8c26cd1d71b3d53d92d03da7b5bf
Lung_Opacity-5338.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5338.png,Lung_Opacity/images/Lung_Opacity-5338.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5338.png,Lung_Opacity/masks/Lung_Opacity-5338.png,True,False,299,299,L,117.76,d7def3c291ee0778727fb1b3db2b7d5e
Lung_Opacity-5339.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5339.png,Lung_Opacity/images/Lung_Opacity-5339.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5339.png,Lung_Opacity/masks/Lung_Opacity-5339.png,True,False,299,299,L,92.54,008e67da7fe49a40244f3e18ccb398af
Lung_Opacity-534.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-534.png,Lung_Opacity/images/Lung_Opacity-534.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-534.png,Lung_Opacity/masks/Lung_Opacity-534.png,True,False,299,299,L,139.79,bc2acc4bcab0e41247231f0e67762e03
Lung_Opacity-5340.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5340.png,Lung_Opacity/images/Lung_Opacity-5340.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5340.png,Lung_Opacity/masks/Lung_Opacity-5340.png,True,False,299,299,L,88.97,e42261bf95ce3c9b4145c582f156c353
Lung_Opacity-5341.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5341.png,Lung_Opacity/images/Lung_Opacity-5341.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5341.png,Lung_Opacity/masks/Lung_Opacity-5341.png,True,False,299,299,L,121.74,e861520139a1d500cb5045d9eac07337
Lung_Opacity-5342.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5342.png,Lung_Opacity/images/Lung_Opacity-5342.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5342.png,Lung_Opacity/masks/Lung_Opacity-5342.png,True,False,299,299,L,141.31,c0da9e496e5be153863735b67db6d3db
Lung_Opacity-5343.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5343.png,Lung_Opacity/images/Lung_Opacity-5343.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5343.png,Lung_Opacity/masks/Lung_Opacity-5343.png,True,False,299,299,L,113.56,7a3303c8ee81eaca64c9572b0d65d2a1
Lung_Opacity-5344.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5344.png,Lung_Opacity/images/Lung_Opacity-5344.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5344.png,Lung_Opacity/masks/Lung_Opacity-5344.png,True,False,299,299,L,110.88,eac3d5a2ca312082a97a9b3846673dc6
Lung_Opacity-5345.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5345.png,Lung_Opacity/images/Lung_Opacity-5345.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5345.png,Lung_Opacity/masks/Lung_Opacity-5345.png,True,False,299,299,L,104.03,593e3781cfa92a4dcc81a8645dd2b169
Lung_Opacity-5346.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5346.png,Lung_Opacity/images/Lung_Opacity-5346.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5346.png,Lung_Opacity/masks/Lung_Opacity-5346.png,True,False,299,299,L,121.22,de8cb2d721139a624669b2b09ab5e132
Lung_Opacity-5347.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5347.png,Lung_Opacity/images/Lung_Opacity-5347.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5347.png,Lung_Opacity/masks/Lung_Opacity-5347.png,True,False,299,299,L,156.9,a45a0e366946a0321fb91e34ebcaa5d7
Lung_Opacity-5348.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5348.png,Lung_Opacity/images/Lung_Opacity-5348.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5348.png,Lung_Opacity/masks/Lung_Opacity-5348.png,True,False,299,299,L,112.67,e9f4de2c704bdf92ed841c763dc4aebe
Lung_Opacity-5349.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5349.png,Lung_Opacity/images/Lung_Opacity-5349.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5349.png,Lung_Opacity/masks/Lung_Opacity-5349.png,True,False,299,299,L,127.51,16d7f622fd13fc8077cc2ce9669b3827
Lung_Opacity-535.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-535.png,Lung_Opacity/images/Lung_Opacity-535.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-535.png,Lung_Opacity/masks/Lung_Opacity-535.png,True,False,299,299,L,118.11,7f3e8d3d61070cb01e030b5209b2147d
Lung_Opacity-5350.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5350.png,Lung_Opacity/images/Lung_Opacity-5350.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5350.png,Lung_Opacity/masks/Lung_Opacity-5350.png,True,False,299,299,L,115.96,5ffe75f162c0799fddd5d989592e678a
Lung_Opacity-5351.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5351.png,Lung_Opacity/images/Lung_Opacity-5351.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5351.png,Lung_Opacity/masks/Lung_Opacity-5351.png,True,False,299,299,L,122.33,e88879471ad727fc86f0782915f3a711
Lung_Opacity-5352.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5352.png,Lung_Opacity/images/Lung_Opacity-5352.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5352.png,Lung_Opacity/masks/Lung_Opacity-5352.png,True,False,299,299,L,133.08,cb4b988734ed8efab461fc134ffd88f1
Lung_Opacity-5353.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5353.png,Lung_Opacity/images/Lung_Opacity-5353.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5353.png,Lung_Opacity/masks/Lung_Opacity-5353.png,True,False,299,299,L,120.34,20ee517ea52acabbfe33648371f41fc3
Lung_Opacity-5354.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5354.png,Lung_Opacity/images/Lung_Opacity-5354.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5354.png,Lung_Opacity/masks/Lung_Opacity-5354.png,True,False,299,299,L,110.15,e423386dfb9041e28fb76830e5ea1d82
Lung_Opacity-5355.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5355.png,Lung_Opacity/images/Lung_Opacity-5355.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5355.png,Lung_Opacity/masks/Lung_Opacity-5355.png,True,False,299,299,L,116.78,2804a5a614ec3b49323a7bc76b8ed12c
Lung_Opacity-5356.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5356.png,Lung_Opacity/images/Lung_Opacity-5356.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5356.png,Lung_Opacity/masks/Lung_Opacity-5356.png,True,False,299,299,L,105.48,1a3ca7a808605fe20856ddbb180a35a4
Lung_Opacity-5357.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5357.png,Lung_Opacity/images/Lung_Opacity-5357.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5357.png,Lung_Opacity/masks/Lung_Opacity-5357.png,True,False,299,299,L,124.07,56dbc3f838d994537a13b69b22a037ed
Lung_Opacity-5358.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5358.png,Lung_Opacity/images/Lung_Opacity-5358.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5358.png,Lung_Opacity/masks/Lung_Opacity-5358.png,True,False,299,299,L,162.78,af1239da67b527b65e71cb0e731fbd9b
Lung_Opacity-5359.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5359.png,Lung_Opacity/images/Lung_Opacity-5359.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5359.png,Lung_Opacity/masks/Lung_Opacity-5359.png,True,False,299,299,L,130.4,d19edeffb4de2535dabb31540497f9fc
Lung_Opacity-536.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-536.png,Lung_Opacity/images/Lung_Opacity-536.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-536.png,Lung_Opacity/masks/Lung_Opacity-536.png,True,False,299,299,L,142.1,d0166840251a7b60e60ee4836bdfce0c
Lung_Opacity-5360.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5360.png,Lung_Opacity/images/Lung_Opacity-5360.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5360.png,Lung_Opacity/masks/Lung_Opacity-5360.png,True,False,299,299,L,113.55,35e4dde17f71a76e1bc5c4bc24a3fa2d
Lung_Opacity-5361.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5361.png,Lung_Opacity/images/Lung_Opacity-5361.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5361.png,Lung_Opacity/masks/Lung_Opacity-5361.png,True,False,299,299,L,89.16,ea35b9ebc65e9b884ec37a12c8ae3f82
Lung_Opacity-5362.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5362.png,Lung_Opacity/images/Lung_Opacity-5362.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5362.png,Lung_Opacity/masks/Lung_Opacity-5362.png,True,False,299,299,L,158.62,dd02c041616f85507ce8c441ce9d3c8a
Lung_Opacity-5363.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5363.png,Lung_Opacity/images/Lung_Opacity-5363.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5363.png,Lung_Opacity/masks/Lung_Opacity-5363.png,True,False,299,299,L,124.09,78e2c06a5487f54152045778b356dce1
Lung_Opacity-5364.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5364.png,Lung_Opacity/images/Lung_Opacity-5364.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5364.png,Lung_Opacity/masks/Lung_Opacity-5364.png,True,False,299,299,L,155.15,9c44a69a93ae403d824f09e89eaf2984
Lung_Opacity-5365.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5365.png,Lung_Opacity/images/Lung_Opacity-5365.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5365.png,Lung_Opacity/masks/Lung_Opacity-5365.png,True,False,299,299,L,107.8,b94eacb3dd7753ccbb28f1e448369f05
Lung_Opacity-5366.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5366.png,Lung_Opacity/images/Lung_Opacity-5366.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5366.png,Lung_Opacity/masks/Lung_Opacity-5366.png,True,False,299,299,L,154.73,cd63f8aa41b3282344baa04541b99035
Lung_Opacity-5367.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5367.png,Lung_Opacity/images/Lung_Opacity-5367.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5367.png,Lung_Opacity/masks/Lung_Opacity-5367.png,True,False,299,299,L,113.28,a4f8ef13745c124f25f5b70845761f69
Lung_Opacity-5368.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5368.png,Lung_Opacity/images/Lung_Opacity-5368.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5368.png,Lung_Opacity/masks/Lung_Opacity-5368.png,True,False,299,299,L,172.74,3d120fb5c02e9bd42362696c9767a988
Lung_Opacity-5369.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5369.png,Lung_Opacity/images/Lung_Opacity-5369.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5369.png,Lung_Opacity/masks/Lung_Opacity-5369.png,True,False,299,299,L,167.63,3bce09093911a6843995b1e575df76e0
Lung_Opacity-537.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-537.png,Lung_Opacity/images/Lung_Opacity-537.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-537.png,Lung_Opacity/masks/Lung_Opacity-537.png,True,False,299,299,L,125.53,e58e87ec8244fb7903e770efdc01b73a
Lung_Opacity-5370.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5370.png,Lung_Opacity/images/Lung_Opacity-5370.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5370.png,Lung_Opacity/masks/Lung_Opacity-5370.png,True,False,299,299,L,139.86,2ecafb3586d60d10847b8436a99ecbc8
Lung_Opacity-5371.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5371.png,Lung_Opacity/images/Lung_Opacity-5371.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5371.png,Lung_Opacity/masks/Lung_Opacity-5371.png,True,False,299,299,L,153.52,6741e94c7ea6cf9aa7a902a9bcf8423a
Lung_Opacity-5372.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5372.png,Lung_Opacity/images/Lung_Opacity-5372.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5372.png,Lung_Opacity/masks/Lung_Opacity-5372.png,True,False,299,299,L,106.82,1f3791ff703595f717848edfcec9645c
Lung_Opacity-5373.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5373.png,Lung_Opacity/images/Lung_Opacity-5373.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5373.png,Lung_Opacity/masks/Lung_Opacity-5373.png,True,False,299,299,L,124.74,238c1bb0cdc884eab121ad2c5884e8cf
Lung_Opacity-5374.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5374.png,Lung_Opacity/images/Lung_Opacity-5374.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5374.png,Lung_Opacity/masks/Lung_Opacity-5374.png,True,False,299,299,L,53.77,96fa54dd1193a80a6a7ce2276cd6764f
Lung_Opacity-5375.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5375.png,Lung_Opacity/images/Lung_Opacity-5375.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5375.png,Lung_Opacity/masks/Lung_Opacity-5375.png,True,False,299,299,L,104.64,e32b35f0f3bf109250c2f0a9a366c229
Lung_Opacity-5376.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5376.png,Lung_Opacity/images/Lung_Opacity-5376.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5376.png,Lung_Opacity/masks/Lung_Opacity-5376.png,True,False,299,299,L,147.99,885dbd6b2afc431e1bfd532994529516
Lung_Opacity-5377.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5377.png,Lung_Opacity/images/Lung_Opacity-5377.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5377.png,Lung_Opacity/masks/Lung_Opacity-5377.png,True,False,299,299,L,190.51,30bc45846275d5efa80ea932edb08025
Lung_Opacity-5378.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5378.png,Lung_Opacity/images/Lung_Opacity-5378.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5378.png,Lung_Opacity/masks/Lung_Opacity-5378.png,True,False,299,299,L,114.59,93cc6c3f63cc8fb899be5a00cec50c3e
Lung_Opacity-5379.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5379.png,Lung_Opacity/images/Lung_Opacity-5379.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5379.png,Lung_Opacity/masks/Lung_Opacity-5379.png,True,False,299,299,L,113.32,7b7fad0f79d445946253d034dba69007
Lung_Opacity-538.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-538.png,Lung_Opacity/images/Lung_Opacity-538.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-538.png,Lung_Opacity/masks/Lung_Opacity-538.png,True,False,299,299,L,115.4,00ffe7d507d6a5e09fe49eccad51dd98
Lung_Opacity-5380.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5380.png,Lung_Opacity/images/Lung_Opacity-5380.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5380.png,Lung_Opacity/masks/Lung_Opacity-5380.png,True,False,299,299,L,92.07,fd09acfda7dfe50c164013a8371cc5d2
Lung_Opacity-5381.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5381.png,Lung_Opacity/images/Lung_Opacity-5381.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5381.png,Lung_Opacity/masks/Lung_Opacity-5381.png,True,False,299,299,L,153.23,c1cc0db2a748163b737b1cb6dd8546f5
Lung_Opacity-5382.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5382.png,Lung_Opacity/images/Lung_Opacity-5382.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5382.png,Lung_Opacity/masks/Lung_Opacity-5382.png,True,False,299,299,L,102.59,4d41341cc64bb6aebe75908f8702b37a
Lung_Opacity-5383.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5383.png,Lung_Opacity/images/Lung_Opacity-5383.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5383.png,Lung_Opacity/masks/Lung_Opacity-5383.png,True,False,299,299,L,133.6,4beaa21f3987f4e7cedbb0d211a7ed0c
Lung_Opacity-5384.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5384.png,Lung_Opacity/images/Lung_Opacity-5384.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5384.png,Lung_Opacity/masks/Lung_Opacity-5384.png,True,False,299,299,L,153.58,52c14a6c76ea8a377e41f6ce7b717bcb
Lung_Opacity-5385.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5385.png,Lung_Opacity/images/Lung_Opacity-5385.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5385.png,Lung_Opacity/masks/Lung_Opacity-5385.png,True,False,299,299,L,122.9,78f93ac7e7579738f7800552ee46b679
Lung_Opacity-5386.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5386.png,Lung_Opacity/images/Lung_Opacity-5386.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5386.png,Lung_Opacity/masks/Lung_Opacity-5386.png,True,False,299,299,L,104.71,01ae80743ddb6f332ac9dc8379cc3ec6
Lung_Opacity-5387.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5387.png,Lung_Opacity/images/Lung_Opacity-5387.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5387.png,Lung_Opacity/masks/Lung_Opacity-5387.png,True,False,299,299,L,183.01,7717f178509c1153e89529c510a56cb1
Lung_Opacity-5388.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5388.png,Lung_Opacity/images/Lung_Opacity-5388.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5388.png,Lung_Opacity/masks/Lung_Opacity-5388.png,True,False,299,299,L,106.67,7fbb3326d31717ad91b896f0516b3e89
Lung_Opacity-5389.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5389.png,Lung_Opacity/images/Lung_Opacity-5389.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5389.png,Lung_Opacity/masks/Lung_Opacity-5389.png,True,False,299,299,L,118.8,34799b5936c077fc82e40a28ec850727
Lung_Opacity-539.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-539.png,Lung_Opacity/images/Lung_Opacity-539.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-539.png,Lung_Opacity/masks/Lung_Opacity-539.png,True,False,299,299,L,97.99,1e5f8a63a30eec82bf74d136cb7f0166
Lung_Opacity-5390.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5390.png,Lung_Opacity/images/Lung_Opacity-5390.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5390.png,Lung_Opacity/masks/Lung_Opacity-5390.png,True,False,299,299,L,116.59,78c16c14a99873bad55bcf9aeaa7737d
Lung_Opacity-5391.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5391.png,Lung_Opacity/images/Lung_Opacity-5391.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5391.png,Lung_Opacity/masks/Lung_Opacity-5391.png,True,False,299,299,L,116.81,a15fce51e8d2a47d02ae12db8170f34b
Lung_Opacity-5392.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5392.png,Lung_Opacity/images/Lung_Opacity-5392.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5392.png,Lung_Opacity/masks/Lung_Opacity-5392.png,True,False,299,299,L,123.43,5841d74be91f07c72127e4ddca2a5bc9
Lung_Opacity-5393.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5393.png,Lung_Opacity/images/Lung_Opacity-5393.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5393.png,Lung_Opacity/masks/Lung_Opacity-5393.png,True,False,299,299,L,128.68,dae7600695c25c6fc6d702058132e6ee
Lung_Opacity-5394.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5394.png,Lung_Opacity/images/Lung_Opacity-5394.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5394.png,Lung_Opacity/masks/Lung_Opacity-5394.png,True,False,299,299,L,128.76,d32d25581fab3aeb65ae58e08ed468c0
Lung_Opacity-5395.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5395.png,Lung_Opacity/images/Lung_Opacity-5395.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5395.png,Lung_Opacity/masks/Lung_Opacity-5395.png,True,False,299,299,L,130.21,83a362e77bd68fc0b5bea303427abdd9
Lung_Opacity-5396.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5396.png,Lung_Opacity/images/Lung_Opacity-5396.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5396.png,Lung_Opacity/masks/Lung_Opacity-5396.png,True,False,299,299,L,136.91,e4f354c8526ea054bbcb01966d40711e
Lung_Opacity-5397.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5397.png,Lung_Opacity/images/Lung_Opacity-5397.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5397.png,Lung_Opacity/masks/Lung_Opacity-5397.png,True,False,299,299,L,135.17,24d452c2f218d8651279c7a9ee9f057e
Lung_Opacity-5398.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5398.png,Lung_Opacity/images/Lung_Opacity-5398.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5398.png,Lung_Opacity/masks/Lung_Opacity-5398.png,True,False,299,299,L,107.3,02e8c5e7e7181feb53fd0825c84a1ccf
Lung_Opacity-5399.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5399.png,Lung_Opacity/images/Lung_Opacity-5399.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5399.png,Lung_Opacity/masks/Lung_Opacity-5399.png,True,False,299,299,L,118.41,bf591cbd87f7ed6307e2889f743bc60e
Lung_Opacity-54.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-54.png,Lung_Opacity/images/Lung_Opacity-54.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-54.png,Lung_Opacity/masks/Lung_Opacity-54.png,True,False,299,299,L,138.31,6efa74923ab6ee8c7ae344fc35bf1a27
Lung_Opacity-540.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-540.png,Lung_Opacity/images/Lung_Opacity-540.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-540.png,Lung_Opacity/masks/Lung_Opacity-540.png,True,False,299,299,L,113.86,614deb430328ee638eaad3e3d5e0c50d
Lung_Opacity-5400.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5400.png,Lung_Opacity/images/Lung_Opacity-5400.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5400.png,Lung_Opacity/masks/Lung_Opacity-5400.png,True,False,299,299,L,115.6,da17d66da6d1decde441a063a062f38a
Lung_Opacity-5401.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5401.png,Lung_Opacity/images/Lung_Opacity-5401.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5401.png,Lung_Opacity/masks/Lung_Opacity-5401.png,True,False,299,299,L,128.45,66bfc70cbcc45b00a0b1bb166aeefae5
Lung_Opacity-5402.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5402.png,Lung_Opacity/images/Lung_Opacity-5402.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5402.png,Lung_Opacity/masks/Lung_Opacity-5402.png,True,False,299,299,L,150.12,c6b61211f51dd72b293a470f5e2f3766
Lung_Opacity-5403.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5403.png,Lung_Opacity/images/Lung_Opacity-5403.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5403.png,Lung_Opacity/masks/Lung_Opacity-5403.png,True,False,299,299,L,123.97,18c7a226932c25dacdef06dd37d6a567
Lung_Opacity-5404.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5404.png,Lung_Opacity/images/Lung_Opacity-5404.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5404.png,Lung_Opacity/masks/Lung_Opacity-5404.png,True,False,299,299,L,164.66,28d66e6b4a39216ee15139e96dd66c62
Lung_Opacity-5405.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5405.png,Lung_Opacity/images/Lung_Opacity-5405.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5405.png,Lung_Opacity/masks/Lung_Opacity-5405.png,True,False,299,299,L,147.96,cf3f0fabc4a88641411d1cb710df5265
Lung_Opacity-5406.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5406.png,Lung_Opacity/images/Lung_Opacity-5406.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5406.png,Lung_Opacity/masks/Lung_Opacity-5406.png,True,False,299,299,L,127.86,53548697ef22c99079d0beeebb5952fa
Lung_Opacity-5407.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5407.png,Lung_Opacity/images/Lung_Opacity-5407.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5407.png,Lung_Opacity/masks/Lung_Opacity-5407.png,True,False,299,299,L,125.1,d62d9a1c1e5ddd66ee2ef0ce51f67fdc
Lung_Opacity-5408.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5408.png,Lung_Opacity/images/Lung_Opacity-5408.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5408.png,Lung_Opacity/masks/Lung_Opacity-5408.png,True,False,299,299,L,98.46,f64cf3bbff6ed5f0cf930a5f0ebc4dff
Lung_Opacity-5409.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5409.png,Lung_Opacity/images/Lung_Opacity-5409.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5409.png,Lung_Opacity/masks/Lung_Opacity-5409.png,True,False,299,299,L,107.55,7f6bda4ce58986f0e5600fd83442ff11
Lung_Opacity-541.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-541.png,Lung_Opacity/images/Lung_Opacity-541.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-541.png,Lung_Opacity/masks/Lung_Opacity-541.png,True,False,299,299,L,118.19,a38291718f29b754bcab4bb63a2f8c55
Lung_Opacity-5410.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5410.png,Lung_Opacity/images/Lung_Opacity-5410.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5410.png,Lung_Opacity/masks/Lung_Opacity-5410.png,True,False,299,299,L,124.18,a86acf27ac9dad00b2b7b24413174775
Lung_Opacity-5411.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5411.png,Lung_Opacity/images/Lung_Opacity-5411.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5411.png,Lung_Opacity/masks/Lung_Opacity-5411.png,True,False,299,299,L,103.04,69e749142402a87606d7d117c86001b7
Lung_Opacity-5412.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5412.png,Lung_Opacity/images/Lung_Opacity-5412.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5412.png,Lung_Opacity/masks/Lung_Opacity-5412.png,True,False,299,299,L,154.34,fc26153678c2281f57d896d24f55a818
Lung_Opacity-5413.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5413.png,Lung_Opacity/images/Lung_Opacity-5413.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5413.png,Lung_Opacity/masks/Lung_Opacity-5413.png,True,False,299,299,L,110.67,dfbec3c04039ed5a3ae86f688c3bd59e
Lung_Opacity-5414.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5414.png,Lung_Opacity/images/Lung_Opacity-5414.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5414.png,Lung_Opacity/masks/Lung_Opacity-5414.png,True,False,299,299,L,157.21,48f9fb5b0c7b728874fdbc8091c08f67
Lung_Opacity-5415.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5415.png,Lung_Opacity/images/Lung_Opacity-5415.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5415.png,Lung_Opacity/masks/Lung_Opacity-5415.png,True,False,299,299,L,149.85,2ffc657144bd86b87d59c210bfda0990
Lung_Opacity-5416.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5416.png,Lung_Opacity/images/Lung_Opacity-5416.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5416.png,Lung_Opacity/masks/Lung_Opacity-5416.png,True,False,299,299,L,136.27,c89920cb345cd4691e3bdbd9adbaf1a1
Lung_Opacity-5417.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5417.png,Lung_Opacity/images/Lung_Opacity-5417.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5417.png,Lung_Opacity/masks/Lung_Opacity-5417.png,True,False,299,299,L,77.92,ebf88102f09e4cdfa0de63f15bf91be6
Lung_Opacity-5418.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5418.png,Lung_Opacity/images/Lung_Opacity-5418.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5418.png,Lung_Opacity/masks/Lung_Opacity-5418.png,True,False,299,299,L,95.77,7db3114f16c646e0dca044dec164e006
Lung_Opacity-5419.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5419.png,Lung_Opacity/images/Lung_Opacity-5419.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5419.png,Lung_Opacity/masks/Lung_Opacity-5419.png,True,False,299,299,L,169.3,6696721eee993fcd0568498d27058a78
Lung_Opacity-542.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-542.png,Lung_Opacity/images/Lung_Opacity-542.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-542.png,Lung_Opacity/masks/Lung_Opacity-542.png,True,False,299,299,L,144.94,68a18dc40155f0608c6dc71d5e84ca52
Lung_Opacity-5420.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5420.png,Lung_Opacity/images/Lung_Opacity-5420.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5420.png,Lung_Opacity/masks/Lung_Opacity-5420.png,True,False,299,299,L,115.07,673a95922eb6c7ef3123bbb5e0df7ea6
Lung_Opacity-5421.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5421.png,Lung_Opacity/images/Lung_Opacity-5421.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5421.png,Lung_Opacity/masks/Lung_Opacity-5421.png,True,False,299,299,L,146.98,2cc1104244461cc6238a4a0079989338
Lung_Opacity-5422.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5422.png,Lung_Opacity/images/Lung_Opacity-5422.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5422.png,Lung_Opacity/masks/Lung_Opacity-5422.png,True,False,299,299,L,127.35,f3084ea8d11de4b6ea2c0de4595f4318
Lung_Opacity-5423.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5423.png,Lung_Opacity/images/Lung_Opacity-5423.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5423.png,Lung_Opacity/masks/Lung_Opacity-5423.png,True,False,299,299,L,115.71,74280f55e01b8f150c9479b5685864ea
Lung_Opacity-5424.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5424.png,Lung_Opacity/images/Lung_Opacity-5424.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5424.png,Lung_Opacity/masks/Lung_Opacity-5424.png,True,False,299,299,L,126.6,7ea2e1b45b7a4d80a25c82c0921b7705
Lung_Opacity-5425.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5425.png,Lung_Opacity/images/Lung_Opacity-5425.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5425.png,Lung_Opacity/masks/Lung_Opacity-5425.png,True,False,299,299,L,111.2,e0957e73365d63670c4aa0df953f9b5b
Lung_Opacity-5426.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5426.png,Lung_Opacity/images/Lung_Opacity-5426.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5426.png,Lung_Opacity/masks/Lung_Opacity-5426.png,True,False,299,299,L,103.94,e5d9b3ad3e3e77bde46ecde85a4e7546
Lung_Opacity-5427.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5427.png,Lung_Opacity/images/Lung_Opacity-5427.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5427.png,Lung_Opacity/masks/Lung_Opacity-5427.png,True,False,299,299,L,115.89,55e18a36facdc6542d3ad8309e323fcb
Lung_Opacity-5428.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5428.png,Lung_Opacity/images/Lung_Opacity-5428.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5428.png,Lung_Opacity/masks/Lung_Opacity-5428.png,True,False,299,299,L,83.85,ed9bbc771e9bacac76773cb4c0d9d316
Lung_Opacity-5429.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5429.png,Lung_Opacity/images/Lung_Opacity-5429.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5429.png,Lung_Opacity/masks/Lung_Opacity-5429.png,True,False,299,299,L,122.07,b90e46156c740f96451aff67b2c8fff8
Lung_Opacity-543.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-543.png,Lung_Opacity/images/Lung_Opacity-543.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-543.png,Lung_Opacity/masks/Lung_Opacity-543.png,True,False,299,299,L,101.2,a77ec2fd382155d8bef7dd147fa5360d
Lung_Opacity-5430.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5430.png,Lung_Opacity/images/Lung_Opacity-5430.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5430.png,Lung_Opacity/masks/Lung_Opacity-5430.png,True,False,299,299,L,103.15,99537be0a658c4a4aa70f4ab936c73a3
Lung_Opacity-5431.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5431.png,Lung_Opacity/images/Lung_Opacity-5431.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5431.png,Lung_Opacity/masks/Lung_Opacity-5431.png,True,False,299,299,L,114.43,fda5d036ce5bf2ca226fe2994c228d21
Lung_Opacity-5432.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5432.png,Lung_Opacity/images/Lung_Opacity-5432.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5432.png,Lung_Opacity/masks/Lung_Opacity-5432.png,True,False,299,299,L,145.16,f90dfb694677a13361212ce93d8df653
Lung_Opacity-5433.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5433.png,Lung_Opacity/images/Lung_Opacity-5433.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5433.png,Lung_Opacity/masks/Lung_Opacity-5433.png,True,False,299,299,L,170.24,46ec57aa5e52a413129e13760217c605
Lung_Opacity-5434.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5434.png,Lung_Opacity/images/Lung_Opacity-5434.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5434.png,Lung_Opacity/masks/Lung_Opacity-5434.png,True,False,299,299,L,157.22,a646135226d3c526a6c88bf461b53e8b
Lung_Opacity-5435.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5435.png,Lung_Opacity/images/Lung_Opacity-5435.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5435.png,Lung_Opacity/masks/Lung_Opacity-5435.png,True,False,299,299,L,104.92,fafe63952962b29c843458e60dc05227
Lung_Opacity-5436.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5436.png,Lung_Opacity/images/Lung_Opacity-5436.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5436.png,Lung_Opacity/masks/Lung_Opacity-5436.png,True,False,299,299,L,101.24,1cc7f258c0e4fb8c1c9835d0ac4b4fbf
Lung_Opacity-5437.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5437.png,Lung_Opacity/images/Lung_Opacity-5437.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5437.png,Lung_Opacity/masks/Lung_Opacity-5437.png,True,False,299,299,L,92.6,e2b38ce0b444cdd8d5ffb50ed724bcd6
Lung_Opacity-5438.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5438.png,Lung_Opacity/images/Lung_Opacity-5438.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5438.png,Lung_Opacity/masks/Lung_Opacity-5438.png,True,False,299,299,L,121.67,ecb7fd5946f7506f6510a2d4302fc550
Lung_Opacity-5439.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5439.png,Lung_Opacity/images/Lung_Opacity-5439.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5439.png,Lung_Opacity/masks/Lung_Opacity-5439.png,True,False,299,299,L,121.44,44a43415207dff260d1427037dc7b240
Lung_Opacity-544.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-544.png,Lung_Opacity/images/Lung_Opacity-544.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-544.png,Lung_Opacity/masks/Lung_Opacity-544.png,True,False,299,299,L,108.74,0091214b43763737b61590f2ec0f4f82
Lung_Opacity-5440.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5440.png,Lung_Opacity/images/Lung_Opacity-5440.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5440.png,Lung_Opacity/masks/Lung_Opacity-5440.png,True,False,299,299,L,147.52,d8cfa2a7f724a01f1d8b1af4b1659d75
Lung_Opacity-5441.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5441.png,Lung_Opacity/images/Lung_Opacity-5441.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5441.png,Lung_Opacity/masks/Lung_Opacity-5441.png,True,False,299,299,L,149.35,79797dfc677e8b2652347146e7646ffe
Lung_Opacity-5442.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5442.png,Lung_Opacity/images/Lung_Opacity-5442.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5442.png,Lung_Opacity/masks/Lung_Opacity-5442.png,True,False,299,299,L,75.2,73b816109657d6dbcde96f88bd47ad2f
Lung_Opacity-5443.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5443.png,Lung_Opacity/images/Lung_Opacity-5443.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5443.png,Lung_Opacity/masks/Lung_Opacity-5443.png,True,False,299,299,L,116.48,fcc8c3f670a5a3c4e4d18213334df752
Lung_Opacity-5444.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5444.png,Lung_Opacity/images/Lung_Opacity-5444.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5444.png,Lung_Opacity/masks/Lung_Opacity-5444.png,True,False,299,299,L,127.37,a9e1992c0fb7af08826ee5179a1edac7
Lung_Opacity-5445.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5445.png,Lung_Opacity/images/Lung_Opacity-5445.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5445.png,Lung_Opacity/masks/Lung_Opacity-5445.png,True,False,299,299,L,127.81,c752670abe266a5dbe9354a5b657933e
Lung_Opacity-5446.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5446.png,Lung_Opacity/images/Lung_Opacity-5446.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5446.png,Lung_Opacity/masks/Lung_Opacity-5446.png,True,False,299,299,L,56.64,1da78f6c76f381b0fe5fc9172e1d423e
Lung_Opacity-5447.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5447.png,Lung_Opacity/images/Lung_Opacity-5447.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5447.png,Lung_Opacity/masks/Lung_Opacity-5447.png,True,False,299,299,L,118.0,c94fee4d69b0598576dd98b40b4ed10f
Lung_Opacity-5448.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5448.png,Lung_Opacity/images/Lung_Opacity-5448.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5448.png,Lung_Opacity/masks/Lung_Opacity-5448.png,True,False,299,299,L,167.51,8c23a2c9a75f79f6d36c509d01313da3
Lung_Opacity-5449.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5449.png,Lung_Opacity/images/Lung_Opacity-5449.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5449.png,Lung_Opacity/masks/Lung_Opacity-5449.png,True,False,299,299,L,114.02,a908d309b5ef78d88730c5ff6e09ac8e
Lung_Opacity-545.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-545.png,Lung_Opacity/images/Lung_Opacity-545.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-545.png,Lung_Opacity/masks/Lung_Opacity-545.png,True,False,299,299,L,110.58,07d7259548adcaf658d695c139ff4ffb
Lung_Opacity-5450.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5450.png,Lung_Opacity/images/Lung_Opacity-5450.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5450.png,Lung_Opacity/masks/Lung_Opacity-5450.png,True,False,299,299,L,103.16,d4600638b35aaa57c81357eb820bae7a
Lung_Opacity-5451.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5451.png,Lung_Opacity/images/Lung_Opacity-5451.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5451.png,Lung_Opacity/masks/Lung_Opacity-5451.png,True,False,299,299,L,123.55,158981fad2d68574690fef13f390779e
Lung_Opacity-5452.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5452.png,Lung_Opacity/images/Lung_Opacity-5452.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5452.png,Lung_Opacity/masks/Lung_Opacity-5452.png,True,False,299,299,L,147.29,8a719e840202217f38b34d398a46221f
Lung_Opacity-5453.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5453.png,Lung_Opacity/images/Lung_Opacity-5453.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5453.png,Lung_Opacity/masks/Lung_Opacity-5453.png,True,False,299,299,L,117.68,d1d14cd68705a96107ccd785c1c7540a
Lung_Opacity-5454.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5454.png,Lung_Opacity/images/Lung_Opacity-5454.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5454.png,Lung_Opacity/masks/Lung_Opacity-5454.png,True,False,299,299,L,112.48,b68530db93cd8bc5bbbb7c6e9b6005cf
Lung_Opacity-5455.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5455.png,Lung_Opacity/images/Lung_Opacity-5455.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5455.png,Lung_Opacity/masks/Lung_Opacity-5455.png,True,False,299,299,L,112.52,747dfe56326eb576b88ae07a0d6daada
Lung_Opacity-5456.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5456.png,Lung_Opacity/images/Lung_Opacity-5456.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5456.png,Lung_Opacity/masks/Lung_Opacity-5456.png,True,False,299,299,L,152.51,ff2929b8598d0e449fa61b87e94f94fd
Lung_Opacity-5457.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5457.png,Lung_Opacity/images/Lung_Opacity-5457.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5457.png,Lung_Opacity/masks/Lung_Opacity-5457.png,True,False,299,299,L,134.02,53b2e251eb1de26f7e103fbe27041604
Lung_Opacity-5458.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5458.png,Lung_Opacity/images/Lung_Opacity-5458.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5458.png,Lung_Opacity/masks/Lung_Opacity-5458.png,True,False,299,299,L,86.63,c7c6adc634751997105d7be30453aad2
Lung_Opacity-5459.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5459.png,Lung_Opacity/images/Lung_Opacity-5459.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5459.png,Lung_Opacity/masks/Lung_Opacity-5459.png,True,False,299,299,L,133.07,0453b69a60377e18d11e6095c81fe39d
Lung_Opacity-546.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-546.png,Lung_Opacity/images/Lung_Opacity-546.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-546.png,Lung_Opacity/masks/Lung_Opacity-546.png,True,False,299,299,L,84.36,7d80a842e6e7dad51224556dc4f6bf52
Lung_Opacity-5460.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5460.png,Lung_Opacity/images/Lung_Opacity-5460.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5460.png,Lung_Opacity/masks/Lung_Opacity-5460.png,True,False,299,299,L,159.34,8c5b9e595103f2a36aa6b6cd9ab7827c
Lung_Opacity-5461.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5461.png,Lung_Opacity/images/Lung_Opacity-5461.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5461.png,Lung_Opacity/masks/Lung_Opacity-5461.png,True,False,299,299,L,134.61,b79e5e2c2f0a326f0fb1b4520c63e21b
Lung_Opacity-5462.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5462.png,Lung_Opacity/images/Lung_Opacity-5462.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5462.png,Lung_Opacity/masks/Lung_Opacity-5462.png,True,False,299,299,L,172.65,cc56cdae8ed10c112ffd2fbac605d5b0
Lung_Opacity-5463.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5463.png,Lung_Opacity/images/Lung_Opacity-5463.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5463.png,Lung_Opacity/masks/Lung_Opacity-5463.png,True,False,299,299,L,120.74,53c9255dd5739938bd0df9e40f0c2d60
Lung_Opacity-5464.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5464.png,Lung_Opacity/images/Lung_Opacity-5464.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5464.png,Lung_Opacity/masks/Lung_Opacity-5464.png,True,False,299,299,L,110.54,51330f482388dca14a4a636514a563a2
Lung_Opacity-5465.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5465.png,Lung_Opacity/images/Lung_Opacity-5465.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5465.png,Lung_Opacity/masks/Lung_Opacity-5465.png,True,False,299,299,L,119.14,db6987cf93bb5ba5e1e3dac79e0d38da
Lung_Opacity-5466.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5466.png,Lung_Opacity/images/Lung_Opacity-5466.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5466.png,Lung_Opacity/masks/Lung_Opacity-5466.png,True,False,299,299,L,128.52,d98d7cc2506752ae4150d84ce2811581
Lung_Opacity-5467.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5467.png,Lung_Opacity/images/Lung_Opacity-5467.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5467.png,Lung_Opacity/masks/Lung_Opacity-5467.png,True,False,299,299,L,91.65,fc8f4a60ed17d6d2bc70a1cc088809dd
Lung_Opacity-5468.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5468.png,Lung_Opacity/images/Lung_Opacity-5468.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5468.png,Lung_Opacity/masks/Lung_Opacity-5468.png,True,False,299,299,L,128.62,d6ce3a44c1a5f0415524cb911a327dcd
Lung_Opacity-5469.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5469.png,Lung_Opacity/images/Lung_Opacity-5469.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5469.png,Lung_Opacity/masks/Lung_Opacity-5469.png,True,False,299,299,L,125.07,cee848f4a5ed94a7cc63b1d4708f319d
Lung_Opacity-547.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-547.png,Lung_Opacity/images/Lung_Opacity-547.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-547.png,Lung_Opacity/masks/Lung_Opacity-547.png,True,False,299,299,L,149.99,3e1ecda7cdfeaa617ee618ae2bd4eb89
Lung_Opacity-5470.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5470.png,Lung_Opacity/images/Lung_Opacity-5470.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5470.png,Lung_Opacity/masks/Lung_Opacity-5470.png,True,False,299,299,L,106.29,9d420d5ef30e710baaebed91d399b481
Lung_Opacity-5471.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5471.png,Lung_Opacity/images/Lung_Opacity-5471.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5471.png,Lung_Opacity/masks/Lung_Opacity-5471.png,True,False,299,299,L,121.92,aa3122719d826136da2342f3a35ff774
Lung_Opacity-5472.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5472.png,Lung_Opacity/images/Lung_Opacity-5472.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5472.png,Lung_Opacity/masks/Lung_Opacity-5472.png,True,False,299,299,L,161.95,d7ad056ae8b7cbad4696b757f78d290d
Lung_Opacity-5473.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5473.png,Lung_Opacity/images/Lung_Opacity-5473.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5473.png,Lung_Opacity/masks/Lung_Opacity-5473.png,True,False,299,299,L,116.82,ca3ec1c51313b704017dad577b684d52
Lung_Opacity-5474.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5474.png,Lung_Opacity/images/Lung_Opacity-5474.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5474.png,Lung_Opacity/masks/Lung_Opacity-5474.png,True,False,299,299,L,102.72,5c3b4ce72bcf9a564617fcc1c82e8e33
Lung_Opacity-5475.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5475.png,Lung_Opacity/images/Lung_Opacity-5475.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5475.png,Lung_Opacity/masks/Lung_Opacity-5475.png,True,False,299,299,L,137.24,e7f2658c23b8c2d2773bd736feb7b460
Lung_Opacity-5476.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5476.png,Lung_Opacity/images/Lung_Opacity-5476.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5476.png,Lung_Opacity/masks/Lung_Opacity-5476.png,True,False,299,299,L,112.01,9c07f9932602c0109c5004eddbe765ed
Lung_Opacity-5477.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5477.png,Lung_Opacity/images/Lung_Opacity-5477.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5477.png,Lung_Opacity/masks/Lung_Opacity-5477.png,True,False,299,299,L,152.46,0c6616168a917da8bc340e1bb6203af4
Lung_Opacity-5478.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5478.png,Lung_Opacity/images/Lung_Opacity-5478.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5478.png,Lung_Opacity/masks/Lung_Opacity-5478.png,True,False,299,299,L,111.76,ef76d0898b514a644071d57776a5089c
Lung_Opacity-5479.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5479.png,Lung_Opacity/images/Lung_Opacity-5479.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5479.png,Lung_Opacity/masks/Lung_Opacity-5479.png,True,False,299,299,L,117.77,72a9a895baf8e91301279715b1159e2c
Lung_Opacity-548.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-548.png,Lung_Opacity/images/Lung_Opacity-548.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-548.png,Lung_Opacity/masks/Lung_Opacity-548.png,True,False,299,299,L,169.1,30351b17eae747e620368c0246c175c2
Lung_Opacity-5480.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5480.png,Lung_Opacity/images/Lung_Opacity-5480.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5480.png,Lung_Opacity/masks/Lung_Opacity-5480.png,True,False,299,299,L,155.51,d6d7f3dfbd4e8a3b7ed30ce9086568c2
Lung_Opacity-5481.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5481.png,Lung_Opacity/images/Lung_Opacity-5481.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5481.png,Lung_Opacity/masks/Lung_Opacity-5481.png,True,False,299,299,L,140.58,9f37d477d5d2af20ac57bb082c7e5311
Lung_Opacity-5482.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5482.png,Lung_Opacity/images/Lung_Opacity-5482.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5482.png,Lung_Opacity/masks/Lung_Opacity-5482.png,True,False,299,299,L,116.3,8e84d3f814b8dbc4d19f1f7aeec31d26
Lung_Opacity-5483.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5483.png,Lung_Opacity/images/Lung_Opacity-5483.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5483.png,Lung_Opacity/masks/Lung_Opacity-5483.png,True,False,299,299,L,111.68,e47372a30795a691efaa602e8eb09f12
Lung_Opacity-5484.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5484.png,Lung_Opacity/images/Lung_Opacity-5484.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5484.png,Lung_Opacity/masks/Lung_Opacity-5484.png,True,False,299,299,L,157.46,342b3d3b9e1cb8330c5bd52e5d8e4505
Lung_Opacity-5485.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5485.png,Lung_Opacity/images/Lung_Opacity-5485.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5485.png,Lung_Opacity/masks/Lung_Opacity-5485.png,True,False,299,299,L,168.97,ce5da1f56b6e568b5892bf29af7f6ca3
Lung_Opacity-5486.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5486.png,Lung_Opacity/images/Lung_Opacity-5486.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5486.png,Lung_Opacity/masks/Lung_Opacity-5486.png,True,False,299,299,L,137.3,1ed6729776d53525212e24783f97053e
Lung_Opacity-5487.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5487.png,Lung_Opacity/images/Lung_Opacity-5487.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5487.png,Lung_Opacity/masks/Lung_Opacity-5487.png,True,False,299,299,L,59.95,a5c685e13cef4a4269846cdfcf9a01d3
Lung_Opacity-5488.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5488.png,Lung_Opacity/images/Lung_Opacity-5488.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5488.png,Lung_Opacity/masks/Lung_Opacity-5488.png,True,False,299,299,L,124.61,cc10219118532783d3750aa9f96f8ec7
Lung_Opacity-5489.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5489.png,Lung_Opacity/images/Lung_Opacity-5489.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5489.png,Lung_Opacity/masks/Lung_Opacity-5489.png,True,False,299,299,L,132.06,1c48abdeeb8bd9d5d8149ba9d1f81d75
Lung_Opacity-549.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-549.png,Lung_Opacity/images/Lung_Opacity-549.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-549.png,Lung_Opacity/masks/Lung_Opacity-549.png,True,False,299,299,L,85.99,e0a9a6b3d5fdb83d8056839f5585ea01
Lung_Opacity-5490.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5490.png,Lung_Opacity/images/Lung_Opacity-5490.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5490.png,Lung_Opacity/masks/Lung_Opacity-5490.png,True,False,299,299,L,112.27,6d783cd07d1edbcd919f943c4f0be091
Lung_Opacity-5491.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5491.png,Lung_Opacity/images/Lung_Opacity-5491.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5491.png,Lung_Opacity/masks/Lung_Opacity-5491.png,True,False,299,299,L,130.05,34dff4bd7bccec4d6099cc8a0f7e3e18
Lung_Opacity-5492.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5492.png,Lung_Opacity/images/Lung_Opacity-5492.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5492.png,Lung_Opacity/masks/Lung_Opacity-5492.png,True,False,299,299,L,117.55,d2c4927b7d31e89cf14febd35c2c4f03
Lung_Opacity-5493.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5493.png,Lung_Opacity/images/Lung_Opacity-5493.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5493.png,Lung_Opacity/masks/Lung_Opacity-5493.png,True,False,299,299,L,111.95,46fedd41dd307316800632572d49a332
Lung_Opacity-5494.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5494.png,Lung_Opacity/images/Lung_Opacity-5494.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5494.png,Lung_Opacity/masks/Lung_Opacity-5494.png,True,False,299,299,L,106.95,fccc12d468b01c0312e88a80d0fea125
Lung_Opacity-5495.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5495.png,Lung_Opacity/images/Lung_Opacity-5495.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5495.png,Lung_Opacity/masks/Lung_Opacity-5495.png,True,False,299,299,L,117.77,ff62094ce2b920b378331955b8002014
Lung_Opacity-5496.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5496.png,Lung_Opacity/images/Lung_Opacity-5496.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5496.png,Lung_Opacity/masks/Lung_Opacity-5496.png,True,False,299,299,L,114.45,655991b84332335c98989e77f7fe2298
Lung_Opacity-5497.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5497.png,Lung_Opacity/images/Lung_Opacity-5497.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5497.png,Lung_Opacity/masks/Lung_Opacity-5497.png,True,False,299,299,L,119.17,e62de73011ff8c5b72efa99ad2b12ee4
Lung_Opacity-5498.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5498.png,Lung_Opacity/images/Lung_Opacity-5498.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5498.png,Lung_Opacity/masks/Lung_Opacity-5498.png,True,False,299,299,L,99.88,32a902da1b28ab51dcf2f762ce77422e
Lung_Opacity-5499.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5499.png,Lung_Opacity/images/Lung_Opacity-5499.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5499.png,Lung_Opacity/masks/Lung_Opacity-5499.png,True,False,299,299,L,106.37,2cb4b7278a0d2a3058b48e2d8d7d1a2d
Lung_Opacity-55.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-55.png,Lung_Opacity/images/Lung_Opacity-55.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-55.png,Lung_Opacity/masks/Lung_Opacity-55.png,True,False,299,299,L,126.49,6566b56b854ceba2b6856f2b363096a5
Lung_Opacity-550.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-550.png,Lung_Opacity/images/Lung_Opacity-550.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-550.png,Lung_Opacity/masks/Lung_Opacity-550.png,True,False,299,299,L,113.23,108fa792703ba40a992d32cca8d69995
Lung_Opacity-5500.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5500.png,Lung_Opacity/images/Lung_Opacity-5500.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5500.png,Lung_Opacity/masks/Lung_Opacity-5500.png,True,False,299,299,L,127.67,2f8755da5067194cce71c78eb7f4c405
Lung_Opacity-5501.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5501.png,Lung_Opacity/images/Lung_Opacity-5501.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5501.png,Lung_Opacity/masks/Lung_Opacity-5501.png,True,False,299,299,L,119.2,a9460a7e070402d795d336381aa7b964
Lung_Opacity-5502.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5502.png,Lung_Opacity/images/Lung_Opacity-5502.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5502.png,Lung_Opacity/masks/Lung_Opacity-5502.png,True,False,299,299,L,102.32,a306875cc90b0484a9c8d45ac2dd5188
Lung_Opacity-5503.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5503.png,Lung_Opacity/images/Lung_Opacity-5503.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5503.png,Lung_Opacity/masks/Lung_Opacity-5503.png,True,False,299,299,L,149.77,b5417c8255e52d6eb2f99c300dd0bb6d
Lung_Opacity-5504.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5504.png,Lung_Opacity/images/Lung_Opacity-5504.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5504.png,Lung_Opacity/masks/Lung_Opacity-5504.png,True,False,299,299,L,113.88,6c73658686db1642c0cb2c472c04b546
Lung_Opacity-5505.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5505.png,Lung_Opacity/images/Lung_Opacity-5505.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5505.png,Lung_Opacity/masks/Lung_Opacity-5505.png,True,False,299,299,L,159.53,b3581641417a3fd5d6a01e614aa613da
Lung_Opacity-5506.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5506.png,Lung_Opacity/images/Lung_Opacity-5506.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5506.png,Lung_Opacity/masks/Lung_Opacity-5506.png,True,False,299,299,L,144.91,1439a76b31c683917f3a7cd9a938d573
Lung_Opacity-5507.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5507.png,Lung_Opacity/images/Lung_Opacity-5507.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5507.png,Lung_Opacity/masks/Lung_Opacity-5507.png,True,False,299,299,L,161.36,969dbcc2135ba31f47216de7273ad715
Lung_Opacity-5508.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5508.png,Lung_Opacity/images/Lung_Opacity-5508.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5508.png,Lung_Opacity/masks/Lung_Opacity-5508.png,True,False,299,299,L,141.88,f016f68c3fe38d61e2220e96fdc291e6
Lung_Opacity-5509.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5509.png,Lung_Opacity/images/Lung_Opacity-5509.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5509.png,Lung_Opacity/masks/Lung_Opacity-5509.png,True,False,299,299,L,151.09,731d61424b4180bbf2df9df3a0559b8e
Lung_Opacity-551.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-551.png,Lung_Opacity/images/Lung_Opacity-551.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-551.png,Lung_Opacity/masks/Lung_Opacity-551.png,True,False,299,299,L,166.11,dc212ff0d5d08c6e9a33d978b5c90b41
Lung_Opacity-5510.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5510.png,Lung_Opacity/images/Lung_Opacity-5510.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5510.png,Lung_Opacity/masks/Lung_Opacity-5510.png,True,False,299,299,L,114.05,71189bfba73994e361ccd74bba8ee12a
Lung_Opacity-5511.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5511.png,Lung_Opacity/images/Lung_Opacity-5511.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5511.png,Lung_Opacity/masks/Lung_Opacity-5511.png,True,False,299,299,L,100.6,591b5b380c50f37399f734e7d32dae23
Lung_Opacity-5512.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5512.png,Lung_Opacity/images/Lung_Opacity-5512.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5512.png,Lung_Opacity/masks/Lung_Opacity-5512.png,True,False,299,299,L,134.45,f4e1f18ce0ea8c99ff5da398a06805e2
Lung_Opacity-5513.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5513.png,Lung_Opacity/images/Lung_Opacity-5513.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5513.png,Lung_Opacity/masks/Lung_Opacity-5513.png,True,False,299,299,L,112.74,a44e18af0af18e98c231cc3d7d26518a
Lung_Opacity-5514.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5514.png,Lung_Opacity/images/Lung_Opacity-5514.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5514.png,Lung_Opacity/masks/Lung_Opacity-5514.png,True,False,299,299,L,142.24,e3965b97131cf86b32c5e1a7fb3c7d5e
Lung_Opacity-5515.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5515.png,Lung_Opacity/images/Lung_Opacity-5515.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5515.png,Lung_Opacity/masks/Lung_Opacity-5515.png,True,False,299,299,L,131.28,875c1253724e77b8f79f54de1ba89fcd
Lung_Opacity-5516.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5516.png,Lung_Opacity/images/Lung_Opacity-5516.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5516.png,Lung_Opacity/masks/Lung_Opacity-5516.png,True,False,299,299,L,119.67,d68b0f3a61b45783df2d5292eb48486e
Lung_Opacity-5517.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5517.png,Lung_Opacity/images/Lung_Opacity-5517.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5517.png,Lung_Opacity/masks/Lung_Opacity-5517.png,True,False,299,299,L,116.14,7e2fbdd470dc16c2c869d999a5d857ef
Lung_Opacity-5518.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5518.png,Lung_Opacity/images/Lung_Opacity-5518.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5518.png,Lung_Opacity/masks/Lung_Opacity-5518.png,True,False,299,299,L,106.2,333ccc5ae5e814461ef1438aa8008e45
Lung_Opacity-5519.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5519.png,Lung_Opacity/images/Lung_Opacity-5519.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5519.png,Lung_Opacity/masks/Lung_Opacity-5519.png,True,False,299,299,L,101.94,ef648191a646ee4329c3a55af77560cb
Lung_Opacity-552.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-552.png,Lung_Opacity/images/Lung_Opacity-552.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-552.png,Lung_Opacity/masks/Lung_Opacity-552.png,True,False,299,299,L,98.8,aae0c85f164845cf60b8911feafa1054
Lung_Opacity-5520.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5520.png,Lung_Opacity/images/Lung_Opacity-5520.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5520.png,Lung_Opacity/masks/Lung_Opacity-5520.png,True,False,299,299,L,114.88,e9dfaedfd2d77f74e83eea2a38d4a6a8
Lung_Opacity-5521.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5521.png,Lung_Opacity/images/Lung_Opacity-5521.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5521.png,Lung_Opacity/masks/Lung_Opacity-5521.png,True,False,299,299,L,143.31,8c8f4e407fb0c2cc4d1be2206da4a15b
Lung_Opacity-5522.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5522.png,Lung_Opacity/images/Lung_Opacity-5522.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5522.png,Lung_Opacity/masks/Lung_Opacity-5522.png,True,False,299,299,L,125.48,59e213e020399fd570a0aaaf78f48d06
Lung_Opacity-5523.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5523.png,Lung_Opacity/images/Lung_Opacity-5523.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5523.png,Lung_Opacity/masks/Lung_Opacity-5523.png,True,False,299,299,L,107.33,52d9883f1129b1df2b089f97eb947634
Lung_Opacity-5524.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5524.png,Lung_Opacity/images/Lung_Opacity-5524.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5524.png,Lung_Opacity/masks/Lung_Opacity-5524.png,True,False,299,299,L,127.29,8bac48b60d1deffd57534fb9ac27c711
Lung_Opacity-5525.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5525.png,Lung_Opacity/images/Lung_Opacity-5525.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5525.png,Lung_Opacity/masks/Lung_Opacity-5525.png,True,False,299,299,L,107.19,0652caa22ba41503670ea01bd5c4ac08
Lung_Opacity-5526.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5526.png,Lung_Opacity/images/Lung_Opacity-5526.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5526.png,Lung_Opacity/masks/Lung_Opacity-5526.png,True,False,299,299,L,143.13,49d7d4d3828dbb130c757905ad59434b
Lung_Opacity-5527.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5527.png,Lung_Opacity/images/Lung_Opacity-5527.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5527.png,Lung_Opacity/masks/Lung_Opacity-5527.png,True,False,299,299,L,148.32,72e4985aec5bc8929e62c9d979c6d8fb
Lung_Opacity-5528.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5528.png,Lung_Opacity/images/Lung_Opacity-5528.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5528.png,Lung_Opacity/masks/Lung_Opacity-5528.png,True,False,299,299,L,126.95,a7f35ce46b419c5cb7895eb90a2e7b8d
Lung_Opacity-5529.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5529.png,Lung_Opacity/images/Lung_Opacity-5529.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5529.png,Lung_Opacity/masks/Lung_Opacity-5529.png,True,False,299,299,L,149.48,ceecfac76106574652b463e4b8e1add0
Lung_Opacity-553.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-553.png,Lung_Opacity/images/Lung_Opacity-553.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-553.png,Lung_Opacity/masks/Lung_Opacity-553.png,True,False,299,299,L,115.21,2ff47ac6b7f85bf34b2d41452f2c4f4a
Lung_Opacity-5530.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5530.png,Lung_Opacity/images/Lung_Opacity-5530.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5530.png,Lung_Opacity/masks/Lung_Opacity-5530.png,True,False,299,299,L,161.59,a76da0465b67cd66de4fa3fe67b435a0
Lung_Opacity-5531.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5531.png,Lung_Opacity/images/Lung_Opacity-5531.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5531.png,Lung_Opacity/masks/Lung_Opacity-5531.png,True,False,299,299,L,110.55,d5a08d72ec937f20d7278fbe56a31a2b
Lung_Opacity-5532.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5532.png,Lung_Opacity/images/Lung_Opacity-5532.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5532.png,Lung_Opacity/masks/Lung_Opacity-5532.png,True,False,299,299,L,101.7,3dae91f2d9d4b98545bcabbdd69e97d3
Lung_Opacity-5533.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5533.png,Lung_Opacity/images/Lung_Opacity-5533.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5533.png,Lung_Opacity/masks/Lung_Opacity-5533.png,True,False,299,299,L,105.23,cee8be8d2b0b86862d476334bf4dfe48
Lung_Opacity-5534.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5534.png,Lung_Opacity/images/Lung_Opacity-5534.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5534.png,Lung_Opacity/masks/Lung_Opacity-5534.png,True,False,299,299,L,126.57,8b4ebdf54fd28907a1e0773924b12799
Lung_Opacity-5535.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5535.png,Lung_Opacity/images/Lung_Opacity-5535.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5535.png,Lung_Opacity/masks/Lung_Opacity-5535.png,True,False,299,299,L,150.17,1081ab4a8b7c6ce32f14849556fbb857
Lung_Opacity-5536.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5536.png,Lung_Opacity/images/Lung_Opacity-5536.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5536.png,Lung_Opacity/masks/Lung_Opacity-5536.png,True,False,299,299,L,169.28,029c6200c020b0d1dd8a6547e95d0e3c
Lung_Opacity-5537.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5537.png,Lung_Opacity/images/Lung_Opacity-5537.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5537.png,Lung_Opacity/masks/Lung_Opacity-5537.png,True,False,299,299,L,105.08,d4458f33a8dcbaa188b6d9a120094a6b
Lung_Opacity-5538.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5538.png,Lung_Opacity/images/Lung_Opacity-5538.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5538.png,Lung_Opacity/masks/Lung_Opacity-5538.png,True,False,299,299,L,104.06,0ac0687c00ca3385e5ddd247a163f2d0
Lung_Opacity-5539.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5539.png,Lung_Opacity/images/Lung_Opacity-5539.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5539.png,Lung_Opacity/masks/Lung_Opacity-5539.png,True,False,299,299,L,103.61,86d1d8e78c63e487de56b4ed8eac6f37
Lung_Opacity-554.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-554.png,Lung_Opacity/images/Lung_Opacity-554.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-554.png,Lung_Opacity/masks/Lung_Opacity-554.png,True,False,299,299,L,113.17,b27f6bb0a449830300a69f9fea6277d0
Lung_Opacity-5540.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5540.png,Lung_Opacity/images/Lung_Opacity-5540.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5540.png,Lung_Opacity/masks/Lung_Opacity-5540.png,True,False,299,299,L,126.7,d4f03fa9953ebf94c9cfffca149ddd4b
Lung_Opacity-5541.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5541.png,Lung_Opacity/images/Lung_Opacity-5541.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5541.png,Lung_Opacity/masks/Lung_Opacity-5541.png,True,False,299,299,L,128.45,f79d50b4cda9d47ad40350dbe23e2d44
Lung_Opacity-5542.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5542.png,Lung_Opacity/images/Lung_Opacity-5542.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5542.png,Lung_Opacity/masks/Lung_Opacity-5542.png,True,False,299,299,L,93.43,932501126f285345799f44c5e457ac46
Lung_Opacity-5543.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5543.png,Lung_Opacity/images/Lung_Opacity-5543.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5543.png,Lung_Opacity/masks/Lung_Opacity-5543.png,True,False,299,299,L,110.06,dec42b112731a4809dce37025f400e58
Lung_Opacity-5544.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5544.png,Lung_Opacity/images/Lung_Opacity-5544.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5544.png,Lung_Opacity/masks/Lung_Opacity-5544.png,True,False,299,299,L,169.09,73124fa01df1b5f650b75bbd1159a8b6
Lung_Opacity-5545.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5545.png,Lung_Opacity/images/Lung_Opacity-5545.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5545.png,Lung_Opacity/masks/Lung_Opacity-5545.png,True,False,299,299,L,111.83,83d092ff7fd8aff8a9a4f00f7b028eba
Lung_Opacity-5546.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5546.png,Lung_Opacity/images/Lung_Opacity-5546.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5546.png,Lung_Opacity/masks/Lung_Opacity-5546.png,True,False,299,299,L,169.02,10fbbb13dcdc0076d2c376ddce368edf
Lung_Opacity-5547.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5547.png,Lung_Opacity/images/Lung_Opacity-5547.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5547.png,Lung_Opacity/masks/Lung_Opacity-5547.png,True,False,299,299,L,112.12,b213c6476bba89c1139bc79b770fac39
Lung_Opacity-5548.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5548.png,Lung_Opacity/images/Lung_Opacity-5548.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5548.png,Lung_Opacity/masks/Lung_Opacity-5548.png,True,False,299,299,L,115.17,77fb624b316a1d5c326bd21730cc853f
Lung_Opacity-5549.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5549.png,Lung_Opacity/images/Lung_Opacity-5549.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5549.png,Lung_Opacity/masks/Lung_Opacity-5549.png,True,False,299,299,L,120.88,92aaa9c3e99036365dcc30b3dacee48e
Lung_Opacity-555.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-555.png,Lung_Opacity/images/Lung_Opacity-555.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-555.png,Lung_Opacity/masks/Lung_Opacity-555.png,True,False,299,299,L,106.72,26a408a43ded1ddfbc624f0b50ea5c27
Lung_Opacity-5550.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5550.png,Lung_Opacity/images/Lung_Opacity-5550.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5550.png,Lung_Opacity/masks/Lung_Opacity-5550.png,True,False,299,299,L,103.56,e156e4ffb0a978ea8d2c8c74833404c3
Lung_Opacity-5551.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5551.png,Lung_Opacity/images/Lung_Opacity-5551.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5551.png,Lung_Opacity/masks/Lung_Opacity-5551.png,True,False,299,299,L,66.15,addc1fe16f5af19ebcf278e12b907287
Lung_Opacity-5552.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5552.png,Lung_Opacity/images/Lung_Opacity-5552.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5552.png,Lung_Opacity/masks/Lung_Opacity-5552.png,True,False,299,299,L,121.49,80e717e69e01d665c2ae3e41a4fb60a1
Lung_Opacity-5553.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5553.png,Lung_Opacity/images/Lung_Opacity-5553.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5553.png,Lung_Opacity/masks/Lung_Opacity-5553.png,True,False,299,299,L,133.5,ebff253a52ae4f96bd163c266e7876ef
Lung_Opacity-5554.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5554.png,Lung_Opacity/images/Lung_Opacity-5554.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5554.png,Lung_Opacity/masks/Lung_Opacity-5554.png,True,False,299,299,L,119.64,082173f42e917cef45e8c4da4bf2eee0
Lung_Opacity-5555.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5555.png,Lung_Opacity/images/Lung_Opacity-5555.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5555.png,Lung_Opacity/masks/Lung_Opacity-5555.png,True,False,299,299,L,109.78,bc6b558d090e9c695c7c3275ee7cebc2
Lung_Opacity-5556.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5556.png,Lung_Opacity/images/Lung_Opacity-5556.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5556.png,Lung_Opacity/masks/Lung_Opacity-5556.png,True,False,299,299,L,140.03,7ed41492969c8f9e3ed36ba490345dc1
Lung_Opacity-5557.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5557.png,Lung_Opacity/images/Lung_Opacity-5557.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5557.png,Lung_Opacity/masks/Lung_Opacity-5557.png,True,False,299,299,L,167.97,77c1cf030e32cd2e4e5bc7631b54a07e
Lung_Opacity-5558.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5558.png,Lung_Opacity/images/Lung_Opacity-5558.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5558.png,Lung_Opacity/masks/Lung_Opacity-5558.png,True,False,299,299,L,152.63,3d7bd9cc320b1161f18c318bf86a91a1
Lung_Opacity-5559.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5559.png,Lung_Opacity/images/Lung_Opacity-5559.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5559.png,Lung_Opacity/masks/Lung_Opacity-5559.png,True,False,299,299,L,131.78,66868d28dbdf3b46a105386de0ef2ec4
Lung_Opacity-556.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-556.png,Lung_Opacity/images/Lung_Opacity-556.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-556.png,Lung_Opacity/masks/Lung_Opacity-556.png,True,False,299,299,L,141.06,17e15e8da3c1eae5b5cd1c9eb01c3bdd
Lung_Opacity-5560.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5560.png,Lung_Opacity/images/Lung_Opacity-5560.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5560.png,Lung_Opacity/masks/Lung_Opacity-5560.png,True,False,299,299,L,158.72,ea8a0a189d2ff0e83b2df03579196e14
Lung_Opacity-5561.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5561.png,Lung_Opacity/images/Lung_Opacity-5561.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5561.png,Lung_Opacity/masks/Lung_Opacity-5561.png,True,False,299,299,L,131.22,adc339906f10cd13e16d87330e901f04
Lung_Opacity-5562.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5562.png,Lung_Opacity/images/Lung_Opacity-5562.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5562.png,Lung_Opacity/masks/Lung_Opacity-5562.png,True,False,299,299,L,142.53,007dfac4073a28aa7e1a2f28323ae8c9
Lung_Opacity-5563.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5563.png,Lung_Opacity/images/Lung_Opacity-5563.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5563.png,Lung_Opacity/masks/Lung_Opacity-5563.png,True,False,299,299,L,113.86,a3b0470c859ae13e928b42e5bc64f108
Lung_Opacity-5564.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5564.png,Lung_Opacity/images/Lung_Opacity-5564.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5564.png,Lung_Opacity/masks/Lung_Opacity-5564.png,True,False,299,299,L,134.35,85beec79216f292010294f2138fc0026
Lung_Opacity-5565.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5565.png,Lung_Opacity/images/Lung_Opacity-5565.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5565.png,Lung_Opacity/masks/Lung_Opacity-5565.png,True,False,299,299,L,135.06,be7420c978d3f67c2b37379418332d6a
Lung_Opacity-5566.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5566.png,Lung_Opacity/images/Lung_Opacity-5566.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5566.png,Lung_Opacity/masks/Lung_Opacity-5566.png,True,False,299,299,L,99.7,33d488b2818094cf2361701cac5b875c
Lung_Opacity-5567.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5567.png,Lung_Opacity/images/Lung_Opacity-5567.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5567.png,Lung_Opacity/masks/Lung_Opacity-5567.png,True,False,299,299,L,110.31,d22e6962663d790806445e715e78c79b
Lung_Opacity-5568.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5568.png,Lung_Opacity/images/Lung_Opacity-5568.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5568.png,Lung_Opacity/masks/Lung_Opacity-5568.png,True,False,299,299,L,101.08,9dbcf68d4094d44a0e6df899240ab307
Lung_Opacity-5569.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5569.png,Lung_Opacity/images/Lung_Opacity-5569.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5569.png,Lung_Opacity/masks/Lung_Opacity-5569.png,True,False,299,299,L,131.64,bdd9c033e4c12528c96d83b97cb0ef0c
Lung_Opacity-557.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-557.png,Lung_Opacity/images/Lung_Opacity-557.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-557.png,Lung_Opacity/masks/Lung_Opacity-557.png,True,False,299,299,L,135.38,167f8f4740d5c6e285976193bbcb533c
Lung_Opacity-5570.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5570.png,Lung_Opacity/images/Lung_Opacity-5570.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5570.png,Lung_Opacity/masks/Lung_Opacity-5570.png,True,False,299,299,L,117.23,14a1f78fe0096e99b86b191375854601
Lung_Opacity-5571.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5571.png,Lung_Opacity/images/Lung_Opacity-5571.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5571.png,Lung_Opacity/masks/Lung_Opacity-5571.png,True,False,299,299,L,165.5,732e7992f3030f0eded9e7edad52a33a
Lung_Opacity-5572.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5572.png,Lung_Opacity/images/Lung_Opacity-5572.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5572.png,Lung_Opacity/masks/Lung_Opacity-5572.png,True,False,299,299,L,51.59,2a3f5bfc413c8254cf4f352083951254
Lung_Opacity-5573.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5573.png,Lung_Opacity/images/Lung_Opacity-5573.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5573.png,Lung_Opacity/masks/Lung_Opacity-5573.png,True,False,299,299,L,50.31,776d7fc196472ef1d221485f7213c72d
Lung_Opacity-5574.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5574.png,Lung_Opacity/images/Lung_Opacity-5574.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5574.png,Lung_Opacity/masks/Lung_Opacity-5574.png,True,False,299,299,L,118.49,e5a44712faa561e72d8ed304dbb4d902
Lung_Opacity-5575.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5575.png,Lung_Opacity/images/Lung_Opacity-5575.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5575.png,Lung_Opacity/masks/Lung_Opacity-5575.png,True,False,299,299,L,103.99,efa4960b8c3281d3d2bbccda605d65e8
Lung_Opacity-5576.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5576.png,Lung_Opacity/images/Lung_Opacity-5576.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5576.png,Lung_Opacity/masks/Lung_Opacity-5576.png,True,False,299,299,L,113.85,97c36183bc20d71612f618d22ffdaad1
Lung_Opacity-5577.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5577.png,Lung_Opacity/images/Lung_Opacity-5577.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5577.png,Lung_Opacity/masks/Lung_Opacity-5577.png,True,False,299,299,L,127.32,ce78a75c4492e0899fd3572085cd3a0c
Lung_Opacity-5578.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5578.png,Lung_Opacity/images/Lung_Opacity-5578.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5578.png,Lung_Opacity/masks/Lung_Opacity-5578.png,True,False,299,299,L,122.86,465d2e79080b0d191330901b98c49310
Lung_Opacity-5579.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5579.png,Lung_Opacity/images/Lung_Opacity-5579.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5579.png,Lung_Opacity/masks/Lung_Opacity-5579.png,True,False,299,299,L,101.08,e02ea3947dbb9e7cd45443b1883e0bc4
Lung_Opacity-558.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-558.png,Lung_Opacity/images/Lung_Opacity-558.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-558.png,Lung_Opacity/masks/Lung_Opacity-558.png,True,False,299,299,L,152.84,d9bff1db78d3591beabe4a8b24298826
Lung_Opacity-5580.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5580.png,Lung_Opacity/images/Lung_Opacity-5580.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5580.png,Lung_Opacity/masks/Lung_Opacity-5580.png,True,False,299,299,L,131.95,d23861fa05ebef627c181ef5054ce304
Lung_Opacity-5581.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5581.png,Lung_Opacity/images/Lung_Opacity-5581.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5581.png,Lung_Opacity/masks/Lung_Opacity-5581.png,True,False,299,299,L,129.41,8996a1f202fe8b967e59ad80b0603526
Lung_Opacity-5582.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5582.png,Lung_Opacity/images/Lung_Opacity-5582.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5582.png,Lung_Opacity/masks/Lung_Opacity-5582.png,True,False,299,299,L,110.75,149fc0c9979150b433eec1a409cd3e30
Lung_Opacity-5583.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5583.png,Lung_Opacity/images/Lung_Opacity-5583.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5583.png,Lung_Opacity/masks/Lung_Opacity-5583.png,True,False,299,299,L,116.11,c69e1955a7ae7641f4b19469c22632b6
Lung_Opacity-5584.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5584.png,Lung_Opacity/images/Lung_Opacity-5584.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5584.png,Lung_Opacity/masks/Lung_Opacity-5584.png,True,False,299,299,L,135.15,e4d79e15441a95bce3b2760829fcfb96
Lung_Opacity-5585.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5585.png,Lung_Opacity/images/Lung_Opacity-5585.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5585.png,Lung_Opacity/masks/Lung_Opacity-5585.png,True,False,299,299,L,117.78,656268680c9d76635c751a1357b39ff1
Lung_Opacity-5586.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5586.png,Lung_Opacity/images/Lung_Opacity-5586.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5586.png,Lung_Opacity/masks/Lung_Opacity-5586.png,True,False,299,299,L,127.52,4ba20ea3aa12052004cf4f3db57cd238
Lung_Opacity-5587.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5587.png,Lung_Opacity/images/Lung_Opacity-5587.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5587.png,Lung_Opacity/masks/Lung_Opacity-5587.png,True,False,299,299,L,117.22,cea957ee06f078b5b6812645b5056e9b
Lung_Opacity-5588.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5588.png,Lung_Opacity/images/Lung_Opacity-5588.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5588.png,Lung_Opacity/masks/Lung_Opacity-5588.png,True,False,299,299,L,111.75,7ab20b59daf390cd84e13c7ba75b9fb3
Lung_Opacity-5589.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5589.png,Lung_Opacity/images/Lung_Opacity-5589.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5589.png,Lung_Opacity/masks/Lung_Opacity-5589.png,True,False,299,299,L,107.79,fb54489312ff1c3e5c8d0c35faec6423
Lung_Opacity-559.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-559.png,Lung_Opacity/images/Lung_Opacity-559.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-559.png,Lung_Opacity/masks/Lung_Opacity-559.png,True,False,299,299,L,134.69,043ef2353bd47e72d9733351cbaaa284
Lung_Opacity-5590.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5590.png,Lung_Opacity/images/Lung_Opacity-5590.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5590.png,Lung_Opacity/masks/Lung_Opacity-5590.png,True,False,299,299,L,157.74,b125aa0472ab6493647809a72b4f44c7
Lung_Opacity-5591.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5591.png,Lung_Opacity/images/Lung_Opacity-5591.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5591.png,Lung_Opacity/masks/Lung_Opacity-5591.png,True,False,299,299,L,74.34,ba0d43833bf10637095d73515d335065
Lung_Opacity-5592.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5592.png,Lung_Opacity/images/Lung_Opacity-5592.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5592.png,Lung_Opacity/masks/Lung_Opacity-5592.png,True,False,299,299,L,136.19,5f893966541a5c0bb522b7f6a89a9276
Lung_Opacity-5593.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5593.png,Lung_Opacity/images/Lung_Opacity-5593.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5593.png,Lung_Opacity/masks/Lung_Opacity-5593.png,True,False,299,299,L,149.67,8f7e08e96ef020afea21e5bd2c4595b0
Lung_Opacity-5594.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5594.png,Lung_Opacity/images/Lung_Opacity-5594.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5594.png,Lung_Opacity/masks/Lung_Opacity-5594.png,True,False,299,299,L,100.23,f50f854fba87fbc8b483db645f883350
Lung_Opacity-5595.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5595.png,Lung_Opacity/images/Lung_Opacity-5595.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5595.png,Lung_Opacity/masks/Lung_Opacity-5595.png,True,False,299,299,L,130.43,389b466c873a8ef4b36767d28119d0bf
Lung_Opacity-5596.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5596.png,Lung_Opacity/images/Lung_Opacity-5596.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5596.png,Lung_Opacity/masks/Lung_Opacity-5596.png,True,False,299,299,L,160.84,3ec96a22f98e92195bf18af07351ee62
Lung_Opacity-5597.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5597.png,Lung_Opacity/images/Lung_Opacity-5597.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5597.png,Lung_Opacity/masks/Lung_Opacity-5597.png,True,False,299,299,L,137.79,1089953435c1496db19124b3a747f2ff
Lung_Opacity-5598.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5598.png,Lung_Opacity/images/Lung_Opacity-5598.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5598.png,Lung_Opacity/masks/Lung_Opacity-5598.png,True,False,299,299,L,112.02,8ad74ce93919fb5f8133c1c6289ebe64
Lung_Opacity-5599.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5599.png,Lung_Opacity/images/Lung_Opacity-5599.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5599.png,Lung_Opacity/masks/Lung_Opacity-5599.png,True,False,299,299,L,152.53,1237f081e420115ca727a880ae33b0c8
Lung_Opacity-56.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-56.png,Lung_Opacity/images/Lung_Opacity-56.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-56.png,Lung_Opacity/masks/Lung_Opacity-56.png,True,False,299,299,L,147.18,82c5d07d15b2b7352c1047a770c66cea
Lung_Opacity-560.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-560.png,Lung_Opacity/images/Lung_Opacity-560.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-560.png,Lung_Opacity/masks/Lung_Opacity-560.png,True,False,299,299,L,109.17,d685c697795a8cd58962202317e1c2c3
Lung_Opacity-5600.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5600.png,Lung_Opacity/images/Lung_Opacity-5600.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5600.png,Lung_Opacity/masks/Lung_Opacity-5600.png,True,False,299,299,L,114.45,b4b76ca431e0bb7c1a66722d3a4cd4fa
Lung_Opacity-5601.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5601.png,Lung_Opacity/images/Lung_Opacity-5601.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5601.png,Lung_Opacity/masks/Lung_Opacity-5601.png,True,False,299,299,L,164.68,63f7db805661438c832b36bdc7a509b3
Lung_Opacity-5602.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5602.png,Lung_Opacity/images/Lung_Opacity-5602.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5602.png,Lung_Opacity/masks/Lung_Opacity-5602.png,True,False,299,299,L,111.54,cb581f579fe385155bde0a18565852bc
Lung_Opacity-5603.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5603.png,Lung_Opacity/images/Lung_Opacity-5603.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5603.png,Lung_Opacity/masks/Lung_Opacity-5603.png,True,False,299,299,L,159.01,da21eaaa6c51f3e15ee4930aeebb4bbd
Lung_Opacity-5604.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5604.png,Lung_Opacity/images/Lung_Opacity-5604.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5604.png,Lung_Opacity/masks/Lung_Opacity-5604.png,True,False,299,299,L,116.96,6455eb936145f6d08b933ef2617d77bd
Lung_Opacity-5605.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5605.png,Lung_Opacity/images/Lung_Opacity-5605.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5605.png,Lung_Opacity/masks/Lung_Opacity-5605.png,True,False,299,299,L,103.73,1e0cfa06d19c9ca07cdc49480fc16e14
Lung_Opacity-5606.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5606.png,Lung_Opacity/images/Lung_Opacity-5606.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5606.png,Lung_Opacity/masks/Lung_Opacity-5606.png,True,False,299,299,L,107.51,956d0b69e47f0cfd531cc7b336a6d487
Lung_Opacity-5607.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5607.png,Lung_Opacity/images/Lung_Opacity-5607.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5607.png,Lung_Opacity/masks/Lung_Opacity-5607.png,True,False,299,299,L,84.89,290f0f0fe84eeef838a7e08600ebcff9
Lung_Opacity-5608.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5608.png,Lung_Opacity/images/Lung_Opacity-5608.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5608.png,Lung_Opacity/masks/Lung_Opacity-5608.png,True,False,299,299,L,129.14,240db1d86ca8da1e97ab243b694c8b72
Lung_Opacity-5609.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5609.png,Lung_Opacity/images/Lung_Opacity-5609.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5609.png,Lung_Opacity/masks/Lung_Opacity-5609.png,True,False,299,299,L,168.52,f625e525f547df8b5c5694880d26ef3c
Lung_Opacity-561.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-561.png,Lung_Opacity/images/Lung_Opacity-561.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-561.png,Lung_Opacity/masks/Lung_Opacity-561.png,True,False,299,299,L,42.47,797d7582359085ef3e2e3820c7efeb4d
Lung_Opacity-5610.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5610.png,Lung_Opacity/images/Lung_Opacity-5610.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5610.png,Lung_Opacity/masks/Lung_Opacity-5610.png,True,False,299,299,L,152.96,dcc6251863f2f53d03147e51ce80e156
Lung_Opacity-5611.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5611.png,Lung_Opacity/images/Lung_Opacity-5611.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5611.png,Lung_Opacity/masks/Lung_Opacity-5611.png,True,False,299,299,L,99.49,0579644adc6d9b242486c282dd7d70e6
Lung_Opacity-5612.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5612.png,Lung_Opacity/images/Lung_Opacity-5612.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5612.png,Lung_Opacity/masks/Lung_Opacity-5612.png,True,False,299,299,L,113.18,6c332b52851656c989b13c2d775f8700
Lung_Opacity-5613.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5613.png,Lung_Opacity/images/Lung_Opacity-5613.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5613.png,Lung_Opacity/masks/Lung_Opacity-5613.png,True,False,299,299,L,118.58,3d9e6c48cf31ee1fb2f4d74cfbf7f36c
Lung_Opacity-5614.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5614.png,Lung_Opacity/images/Lung_Opacity-5614.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5614.png,Lung_Opacity/masks/Lung_Opacity-5614.png,True,False,299,299,L,121.0,f1cf2e9cfbf2686979591f5bbf63fae0
Lung_Opacity-5615.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5615.png,Lung_Opacity/images/Lung_Opacity-5615.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5615.png,Lung_Opacity/masks/Lung_Opacity-5615.png,True,False,299,299,L,162.46,c70c76c62fe8e41cd8a5e130fde0784c
Lung_Opacity-5616.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5616.png,Lung_Opacity/images/Lung_Opacity-5616.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5616.png,Lung_Opacity/masks/Lung_Opacity-5616.png,True,False,299,299,L,111.74,0e4603fa65e39c54c0a3e125421b93b3
Lung_Opacity-5617.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5617.png,Lung_Opacity/images/Lung_Opacity-5617.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5617.png,Lung_Opacity/masks/Lung_Opacity-5617.png,True,False,299,299,L,123.1,d19aa37dac358d910c2d037384688b61
Lung_Opacity-5618.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5618.png,Lung_Opacity/images/Lung_Opacity-5618.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5618.png,Lung_Opacity/masks/Lung_Opacity-5618.png,True,False,299,299,L,166.01,3cada9501d04255e5fc7f5893dd439d1
Lung_Opacity-5619.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5619.png,Lung_Opacity/images/Lung_Opacity-5619.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5619.png,Lung_Opacity/masks/Lung_Opacity-5619.png,True,False,299,299,L,130.16,78c948489ed4b456ad712310f7e67ac8
Lung_Opacity-562.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-562.png,Lung_Opacity/images/Lung_Opacity-562.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-562.png,Lung_Opacity/masks/Lung_Opacity-562.png,True,False,299,299,L,154.17,3e6a5c43ded75f007e87cb662aa1e3b0
Lung_Opacity-5620.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5620.png,Lung_Opacity/images/Lung_Opacity-5620.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5620.png,Lung_Opacity/masks/Lung_Opacity-5620.png,True,False,299,299,L,154.1,a3769811820d96f1b91138e600ec1e5b
Lung_Opacity-5621.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5621.png,Lung_Opacity/images/Lung_Opacity-5621.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5621.png,Lung_Opacity/masks/Lung_Opacity-5621.png,True,False,299,299,L,110.06,44ed368f8400941d483455c1bde57c16
Lung_Opacity-5622.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5622.png,Lung_Opacity/images/Lung_Opacity-5622.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5622.png,Lung_Opacity/masks/Lung_Opacity-5622.png,True,False,299,299,L,113.64,9fd82616c37076ce23776c1ea17d4207
Lung_Opacity-5623.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5623.png,Lung_Opacity/images/Lung_Opacity-5623.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5623.png,Lung_Opacity/masks/Lung_Opacity-5623.png,True,False,299,299,L,94.46,f49a7c3f4c67cc3e13c5c89e7e0a2541
Lung_Opacity-5624.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5624.png,Lung_Opacity/images/Lung_Opacity-5624.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5624.png,Lung_Opacity/masks/Lung_Opacity-5624.png,True,False,299,299,L,107.85,ea0b600ead2ef0068f8f3721ca4524ca
Lung_Opacity-5625.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5625.png,Lung_Opacity/images/Lung_Opacity-5625.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5625.png,Lung_Opacity/masks/Lung_Opacity-5625.png,True,False,299,299,L,141.57,02ac14c19ec9b80030b833d139f754f5
Lung_Opacity-5626.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5626.png,Lung_Opacity/images/Lung_Opacity-5626.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5626.png,Lung_Opacity/masks/Lung_Opacity-5626.png,True,False,299,299,L,98.48,e10b6d63d1e78f95b254dc0ddf241139
Lung_Opacity-5627.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5627.png,Lung_Opacity/images/Lung_Opacity-5627.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5627.png,Lung_Opacity/masks/Lung_Opacity-5627.png,True,False,299,299,L,167.03,3a63804f4abac40e9fa84c349a05fc60
Lung_Opacity-5628.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5628.png,Lung_Opacity/images/Lung_Opacity-5628.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5628.png,Lung_Opacity/masks/Lung_Opacity-5628.png,True,False,299,299,L,121.78,e7e39fc3f37cbe64b215b9849d9f1215
Lung_Opacity-5629.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5629.png,Lung_Opacity/images/Lung_Opacity-5629.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5629.png,Lung_Opacity/masks/Lung_Opacity-5629.png,True,False,299,299,L,121.85,134fa1759d92e95554807ced8b0a14fa
Lung_Opacity-563.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-563.png,Lung_Opacity/images/Lung_Opacity-563.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-563.png,Lung_Opacity/masks/Lung_Opacity-563.png,True,False,299,299,L,111.2,7bb544ec0208a7e682baad2da671ba6e
Lung_Opacity-5630.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5630.png,Lung_Opacity/images/Lung_Opacity-5630.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5630.png,Lung_Opacity/masks/Lung_Opacity-5630.png,True,False,299,299,L,168.12,0e75e711d247d4cd0450632f6d9bb7e5
Lung_Opacity-5631.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5631.png,Lung_Opacity/images/Lung_Opacity-5631.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5631.png,Lung_Opacity/masks/Lung_Opacity-5631.png,True,False,299,299,L,115.61,1b6d1b395c932802916c5568a78fe9c8
Lung_Opacity-5632.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5632.png,Lung_Opacity/images/Lung_Opacity-5632.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5632.png,Lung_Opacity/masks/Lung_Opacity-5632.png,True,False,299,299,L,120.7,e1dc4af2160af1adae20fb178d3350fb
Lung_Opacity-5633.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5633.png,Lung_Opacity/images/Lung_Opacity-5633.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5633.png,Lung_Opacity/masks/Lung_Opacity-5633.png,True,False,299,299,L,134.14,9215e9d757415e8a50bf45d2e67a34f8
Lung_Opacity-5634.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5634.png,Lung_Opacity/images/Lung_Opacity-5634.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5634.png,Lung_Opacity/masks/Lung_Opacity-5634.png,True,False,299,299,L,135.36,a8133b74dc6e8c69195bbc4251a1abe9
Lung_Opacity-5635.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5635.png,Lung_Opacity/images/Lung_Opacity-5635.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5635.png,Lung_Opacity/masks/Lung_Opacity-5635.png,True,False,299,299,L,127.2,92b11dc6a7361609e9c387dde81fc86c
Lung_Opacity-5636.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5636.png,Lung_Opacity/images/Lung_Opacity-5636.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5636.png,Lung_Opacity/masks/Lung_Opacity-5636.png,True,False,299,299,L,123.16,3910e7b65d0ce35e173ec68c0a8f092f
Lung_Opacity-5637.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5637.png,Lung_Opacity/images/Lung_Opacity-5637.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5637.png,Lung_Opacity/masks/Lung_Opacity-5637.png,True,False,299,299,L,115.47,49d96795cea830dcadcac0e5df9d83ac
Lung_Opacity-5638.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5638.png,Lung_Opacity/images/Lung_Opacity-5638.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5638.png,Lung_Opacity/masks/Lung_Opacity-5638.png,True,False,299,299,L,109.25,1707752ec0ebc47b21f4741b65d96013
Lung_Opacity-5639.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5639.png,Lung_Opacity/images/Lung_Opacity-5639.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5639.png,Lung_Opacity/masks/Lung_Opacity-5639.png,True,False,299,299,L,114.25,12105b93eeadcf6b3a388ed861e19559
Lung_Opacity-564.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-564.png,Lung_Opacity/images/Lung_Opacity-564.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-564.png,Lung_Opacity/masks/Lung_Opacity-564.png,True,False,299,299,L,128.12,3fb2a20360f773b2ff1e76b722313ed3
Lung_Opacity-5640.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5640.png,Lung_Opacity/images/Lung_Opacity-5640.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5640.png,Lung_Opacity/masks/Lung_Opacity-5640.png,True,False,299,299,L,139.87,d5da9ac92e0150794451ae1f53e3b168
Lung_Opacity-5641.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5641.png,Lung_Opacity/images/Lung_Opacity-5641.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5641.png,Lung_Opacity/masks/Lung_Opacity-5641.png,True,False,299,299,L,146.5,ec13e219679907cab8845f356aa57c34
Lung_Opacity-5642.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5642.png,Lung_Opacity/images/Lung_Opacity-5642.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5642.png,Lung_Opacity/masks/Lung_Opacity-5642.png,True,False,299,299,L,104.71,d6932b19e8ea6ed7004193b8ec026a69
Lung_Opacity-5643.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5643.png,Lung_Opacity/images/Lung_Opacity-5643.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5643.png,Lung_Opacity/masks/Lung_Opacity-5643.png,True,False,299,299,L,140.92,1631bf6af4a84c167bc29338330c21e2
Lung_Opacity-5644.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5644.png,Lung_Opacity/images/Lung_Opacity-5644.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5644.png,Lung_Opacity/masks/Lung_Opacity-5644.png,True,False,299,299,L,107.71,9135b116b1394f3a0266f7c73c3ee108
Lung_Opacity-5645.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5645.png,Lung_Opacity/images/Lung_Opacity-5645.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5645.png,Lung_Opacity/masks/Lung_Opacity-5645.png,True,False,299,299,L,86.13,57b4c5b72b67de84cf01a8467bb6f93d
Lung_Opacity-5646.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5646.png,Lung_Opacity/images/Lung_Opacity-5646.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5646.png,Lung_Opacity/masks/Lung_Opacity-5646.png,True,False,299,299,L,59.15,2c58e075e31e4587560b0870a73bfc4f
Lung_Opacity-5647.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5647.png,Lung_Opacity/images/Lung_Opacity-5647.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5647.png,Lung_Opacity/masks/Lung_Opacity-5647.png,True,False,299,299,L,84.71,00bc1d83f9f898bdbe5bb34b9b7ed64a
Lung_Opacity-5648.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5648.png,Lung_Opacity/images/Lung_Opacity-5648.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5648.png,Lung_Opacity/masks/Lung_Opacity-5648.png,True,False,299,299,L,85.0,048356e3f76babebd3c2457d9353cc98
Lung_Opacity-5649.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5649.png,Lung_Opacity/images/Lung_Opacity-5649.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5649.png,Lung_Opacity/masks/Lung_Opacity-5649.png,True,False,299,299,L,91.17,8fb21f699e27e60f3c1c087e403cd565
Lung_Opacity-565.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-565.png,Lung_Opacity/images/Lung_Opacity-565.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-565.png,Lung_Opacity/masks/Lung_Opacity-565.png,True,False,299,299,L,111.81,c198a6c29a4eea4fcb27b40bb709641e
Lung_Opacity-5650.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5650.png,Lung_Opacity/images/Lung_Opacity-5650.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5650.png,Lung_Opacity/masks/Lung_Opacity-5650.png,True,False,299,299,L,122.66,302853a60c0ae2828b31be96353c1fb0
Lung_Opacity-5651.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5651.png,Lung_Opacity/images/Lung_Opacity-5651.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5651.png,Lung_Opacity/masks/Lung_Opacity-5651.png,True,False,299,299,L,112.6,2fe0d76f2ca09de04f6fabee0ddea63c
Lung_Opacity-5652.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5652.png,Lung_Opacity/images/Lung_Opacity-5652.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5652.png,Lung_Opacity/masks/Lung_Opacity-5652.png,True,False,299,299,L,116.61,1370d859e50f865f9ddeedb79a1118ec
Lung_Opacity-5653.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5653.png,Lung_Opacity/images/Lung_Opacity-5653.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5653.png,Lung_Opacity/masks/Lung_Opacity-5653.png,True,False,299,299,L,182.49,a8695715fb5490ea36cfda99f720947e
Lung_Opacity-5654.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5654.png,Lung_Opacity/images/Lung_Opacity-5654.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5654.png,Lung_Opacity/masks/Lung_Opacity-5654.png,True,False,299,299,L,94.9,9dcce5c6303eedfc366acba1c46f3324
Lung_Opacity-5655.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5655.png,Lung_Opacity/images/Lung_Opacity-5655.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5655.png,Lung_Opacity/masks/Lung_Opacity-5655.png,True,False,299,299,L,121.09,b39cdf79e779526a9020a818e59c938d
Lung_Opacity-5656.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5656.png,Lung_Opacity/images/Lung_Opacity-5656.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5656.png,Lung_Opacity/masks/Lung_Opacity-5656.png,True,False,299,299,L,103.76,8eff54261653c2f6bd77d46cfcc9ddb3
Lung_Opacity-5657.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5657.png,Lung_Opacity/images/Lung_Opacity-5657.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5657.png,Lung_Opacity/masks/Lung_Opacity-5657.png,True,False,299,299,L,133.72,56ad5b7b44caf0812df541a4db1df4c1
Lung_Opacity-5658.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5658.png,Lung_Opacity/images/Lung_Opacity-5658.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5658.png,Lung_Opacity/masks/Lung_Opacity-5658.png,True,False,299,299,L,116.71,274b8e0d4ff90f85be84528d4a298e99
Lung_Opacity-5659.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5659.png,Lung_Opacity/images/Lung_Opacity-5659.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5659.png,Lung_Opacity/masks/Lung_Opacity-5659.png,True,False,299,299,L,114.95,b8571f98903feeab9e0d317d33d03c31
Lung_Opacity-566.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-566.png,Lung_Opacity/images/Lung_Opacity-566.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-566.png,Lung_Opacity/masks/Lung_Opacity-566.png,True,False,299,299,L,117.72,6a5ec44b7c999d02daf537f1a07e1a8d
Lung_Opacity-5660.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5660.png,Lung_Opacity/images/Lung_Opacity-5660.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5660.png,Lung_Opacity/masks/Lung_Opacity-5660.png,True,False,299,299,L,107.32,ef989106d146f849faab473ca695954c
Lung_Opacity-5661.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5661.png,Lung_Opacity/images/Lung_Opacity-5661.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5661.png,Lung_Opacity/masks/Lung_Opacity-5661.png,True,False,299,299,L,105.31,f922efbe167da55da7bb52bd4262647d
Lung_Opacity-5662.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5662.png,Lung_Opacity/images/Lung_Opacity-5662.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5662.png,Lung_Opacity/masks/Lung_Opacity-5662.png,True,False,299,299,L,102.35,c6786b0a0e723088e0754d4995f64ad9
Lung_Opacity-5663.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5663.png,Lung_Opacity/images/Lung_Opacity-5663.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5663.png,Lung_Opacity/masks/Lung_Opacity-5663.png,True,False,299,299,L,156.0,c06b13a65259814b04bf19c51d35ad1c
Lung_Opacity-5664.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5664.png,Lung_Opacity/images/Lung_Opacity-5664.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5664.png,Lung_Opacity/masks/Lung_Opacity-5664.png,True,False,299,299,L,119.69,67d3a4787396c03b41d1a9fff57e4327
Lung_Opacity-5665.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5665.png,Lung_Opacity/images/Lung_Opacity-5665.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5665.png,Lung_Opacity/masks/Lung_Opacity-5665.png,True,False,299,299,L,122.04,305932bcd5cc0f9e0236e2c8b5b17d3e
Lung_Opacity-5666.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5666.png,Lung_Opacity/images/Lung_Opacity-5666.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5666.png,Lung_Opacity/masks/Lung_Opacity-5666.png,True,False,299,299,L,158.99,ac79ecfecf1a0be6e570315aab8109f2
Lung_Opacity-5667.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5667.png,Lung_Opacity/images/Lung_Opacity-5667.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5667.png,Lung_Opacity/masks/Lung_Opacity-5667.png,True,False,299,299,L,162.69,798f3ea2f84504f1a81e56114ef8fc09
Lung_Opacity-5668.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5668.png,Lung_Opacity/images/Lung_Opacity-5668.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5668.png,Lung_Opacity/masks/Lung_Opacity-5668.png,True,False,299,299,L,103.42,e598292a11f405fc2b9db7df8ed25623
Lung_Opacity-5669.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5669.png,Lung_Opacity/images/Lung_Opacity-5669.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5669.png,Lung_Opacity/masks/Lung_Opacity-5669.png,True,False,299,299,L,132.49,689998afe9c7835a04e7a6064114d862
Lung_Opacity-567.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-567.png,Lung_Opacity/images/Lung_Opacity-567.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-567.png,Lung_Opacity/masks/Lung_Opacity-567.png,True,False,299,299,L,102.63,91db1f47f7f3b364eee8cb4ba644a06a
Lung_Opacity-5670.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5670.png,Lung_Opacity/images/Lung_Opacity-5670.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5670.png,Lung_Opacity/masks/Lung_Opacity-5670.png,True,False,299,299,L,177.19,11f65e0392efdcfd3b7e195c6fffe8c8
Lung_Opacity-5671.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5671.png,Lung_Opacity/images/Lung_Opacity-5671.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5671.png,Lung_Opacity/masks/Lung_Opacity-5671.png,True,False,299,299,L,116.14,48f0082fad7f236e3c99bacc66163022
Lung_Opacity-5672.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5672.png,Lung_Opacity/images/Lung_Opacity-5672.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5672.png,Lung_Opacity/masks/Lung_Opacity-5672.png,True,False,299,299,L,157.51,7c72151af3c9fedfc549a69f0f8770d7
Lung_Opacity-5673.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5673.png,Lung_Opacity/images/Lung_Opacity-5673.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5673.png,Lung_Opacity/masks/Lung_Opacity-5673.png,True,False,299,299,L,160.35,1d4bc821831b493aaefe8149d47f3bfe
Lung_Opacity-5674.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5674.png,Lung_Opacity/images/Lung_Opacity-5674.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5674.png,Lung_Opacity/masks/Lung_Opacity-5674.png,True,False,299,299,L,115.28,ec5a06cbea14bb6b3350b37789ec64f2
Lung_Opacity-5675.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5675.png,Lung_Opacity/images/Lung_Opacity-5675.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5675.png,Lung_Opacity/masks/Lung_Opacity-5675.png,True,False,299,299,L,116.16,15629e0c6eb6660306d84a65d7ab0d0f
Lung_Opacity-5676.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5676.png,Lung_Opacity/images/Lung_Opacity-5676.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5676.png,Lung_Opacity/masks/Lung_Opacity-5676.png,True,False,299,299,L,115.88,2c0535ee8155097d5989e13a10b2e94c
Lung_Opacity-5677.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5677.png,Lung_Opacity/images/Lung_Opacity-5677.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5677.png,Lung_Opacity/masks/Lung_Opacity-5677.png,True,False,299,299,L,117.35,61d29da8ebacb337618cb426d548391e
Lung_Opacity-5678.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5678.png,Lung_Opacity/images/Lung_Opacity-5678.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5678.png,Lung_Opacity/masks/Lung_Opacity-5678.png,True,False,299,299,L,123.06,ccd79bdf7de2e52526ea2e64d7505e89
Lung_Opacity-5679.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5679.png,Lung_Opacity/images/Lung_Opacity-5679.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5679.png,Lung_Opacity/masks/Lung_Opacity-5679.png,True,False,299,299,L,115.0,477a60adaf0f005c8b6b31a2dbb87305
Lung_Opacity-568.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-568.png,Lung_Opacity/images/Lung_Opacity-568.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-568.png,Lung_Opacity/masks/Lung_Opacity-568.png,True,False,299,299,L,126.68,887678713805084f455de94d38117d0a
Lung_Opacity-5680.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5680.png,Lung_Opacity/images/Lung_Opacity-5680.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5680.png,Lung_Opacity/masks/Lung_Opacity-5680.png,True,False,299,299,L,167.69,60f0ce2825700397162cb7bda68058f0
Lung_Opacity-5681.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5681.png,Lung_Opacity/images/Lung_Opacity-5681.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5681.png,Lung_Opacity/masks/Lung_Opacity-5681.png,True,False,299,299,L,101.12,44b0f86f447cff7724ef5bc5e11fce58
Lung_Opacity-5682.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5682.png,Lung_Opacity/images/Lung_Opacity-5682.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5682.png,Lung_Opacity/masks/Lung_Opacity-5682.png,True,False,299,299,L,118.38,4af232a7457dee98e28e22952b889736
Lung_Opacity-5683.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5683.png,Lung_Opacity/images/Lung_Opacity-5683.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5683.png,Lung_Opacity/masks/Lung_Opacity-5683.png,True,False,299,299,L,144.85,8d13893e1d11dfa72ef8d3c1f0919001
Lung_Opacity-5684.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5684.png,Lung_Opacity/images/Lung_Opacity-5684.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5684.png,Lung_Opacity/masks/Lung_Opacity-5684.png,True,False,299,299,L,174.38,35da5a13e083e3dce30fa93385291e94
Lung_Opacity-5685.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5685.png,Lung_Opacity/images/Lung_Opacity-5685.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5685.png,Lung_Opacity/masks/Lung_Opacity-5685.png,True,False,299,299,L,102.84,41db105169810f2630d1946e3b67f924
Lung_Opacity-5686.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5686.png,Lung_Opacity/images/Lung_Opacity-5686.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5686.png,Lung_Opacity/masks/Lung_Opacity-5686.png,True,False,299,299,L,108.22,4cd00567775fce3e05617b2bc0c06dde
Lung_Opacity-5687.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5687.png,Lung_Opacity/images/Lung_Opacity-5687.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5687.png,Lung_Opacity/masks/Lung_Opacity-5687.png,True,False,299,299,L,118.47,1b0a9b45b8a6e417b05b33d93f287bf3
Lung_Opacity-5688.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5688.png,Lung_Opacity/images/Lung_Opacity-5688.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5688.png,Lung_Opacity/masks/Lung_Opacity-5688.png,True,False,299,299,L,116.28,6d9addbc8b726fe82df8c63b81b95bf8
Lung_Opacity-5689.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5689.png,Lung_Opacity/images/Lung_Opacity-5689.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5689.png,Lung_Opacity/masks/Lung_Opacity-5689.png,True,False,299,299,L,59.65,f427c4a661edfe1e9c4bb11bf2a7e3f4
Lung_Opacity-569.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-569.png,Lung_Opacity/images/Lung_Opacity-569.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-569.png,Lung_Opacity/masks/Lung_Opacity-569.png,True,False,299,299,L,163.63,887f4c6c529d9e66441661d071c58dde
Lung_Opacity-5690.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5690.png,Lung_Opacity/images/Lung_Opacity-5690.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5690.png,Lung_Opacity/masks/Lung_Opacity-5690.png,True,False,299,299,L,90.89,2a70bc9094e3883a78372218340e0ef7
Lung_Opacity-5691.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5691.png,Lung_Opacity/images/Lung_Opacity-5691.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5691.png,Lung_Opacity/masks/Lung_Opacity-5691.png,True,False,299,299,L,150.64,fef8974e7ffed9c59258f7da75b38c7e
Lung_Opacity-5692.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5692.png,Lung_Opacity/images/Lung_Opacity-5692.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5692.png,Lung_Opacity/masks/Lung_Opacity-5692.png,True,False,299,299,L,175.4,17480279c5f71583fface45a9a604b74
Lung_Opacity-5693.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5693.png,Lung_Opacity/images/Lung_Opacity-5693.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5693.png,Lung_Opacity/masks/Lung_Opacity-5693.png,True,False,299,299,L,111.25,de7728e27cf7e0ef1069a874e5f89dea
Lung_Opacity-5694.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5694.png,Lung_Opacity/images/Lung_Opacity-5694.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5694.png,Lung_Opacity/masks/Lung_Opacity-5694.png,True,False,299,299,L,128.34,486a5984651fc7551d0c03f01b81010e
Lung_Opacity-5695.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5695.png,Lung_Opacity/images/Lung_Opacity-5695.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5695.png,Lung_Opacity/masks/Lung_Opacity-5695.png,True,False,299,299,L,125.21,543277e5faf49f51815531c0c9ccf23d
Lung_Opacity-5696.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5696.png,Lung_Opacity/images/Lung_Opacity-5696.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5696.png,Lung_Opacity/masks/Lung_Opacity-5696.png,True,False,299,299,L,106.15,e7cc874f23efbaffb0ee3b02710b7130
Lung_Opacity-5697.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5697.png,Lung_Opacity/images/Lung_Opacity-5697.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5697.png,Lung_Opacity/masks/Lung_Opacity-5697.png,True,False,299,299,L,109.77,266fdda28f743c053b35472f73ab0325
Lung_Opacity-5698.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5698.png,Lung_Opacity/images/Lung_Opacity-5698.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5698.png,Lung_Opacity/masks/Lung_Opacity-5698.png,True,False,299,299,L,112.27,e45bde433b21be70d5feea9f9026e22d
Lung_Opacity-5699.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5699.png,Lung_Opacity/images/Lung_Opacity-5699.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5699.png,Lung_Opacity/masks/Lung_Opacity-5699.png,True,False,299,299,L,99.79,4478d9712190b9916c792f282655a215
Lung_Opacity-57.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-57.png,Lung_Opacity/images/Lung_Opacity-57.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-57.png,Lung_Opacity/masks/Lung_Opacity-57.png,True,False,299,299,L,179.36,8d614bd9faac5905bacdc787277a8bc9
Lung_Opacity-570.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-570.png,Lung_Opacity/images/Lung_Opacity-570.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-570.png,Lung_Opacity/masks/Lung_Opacity-570.png,True,False,299,299,L,167.5,2b22eb6f816db577b470b796f2b41053
Lung_Opacity-5700.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5700.png,Lung_Opacity/images/Lung_Opacity-5700.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5700.png,Lung_Opacity/masks/Lung_Opacity-5700.png,True,False,299,299,L,111.81,5b391fa316f0e3f5ef7a8fb924979f65
Lung_Opacity-5701.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5701.png,Lung_Opacity/images/Lung_Opacity-5701.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5701.png,Lung_Opacity/masks/Lung_Opacity-5701.png,True,False,299,299,L,133.03,f2ce4d1ce7bebc2bcbde2dbf58d5140b
Lung_Opacity-5702.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5702.png,Lung_Opacity/images/Lung_Opacity-5702.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5702.png,Lung_Opacity/masks/Lung_Opacity-5702.png,True,False,299,299,L,118.44,ad2d64c3fee7002f3c2372613be3a798
Lung_Opacity-5703.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5703.png,Lung_Opacity/images/Lung_Opacity-5703.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5703.png,Lung_Opacity/masks/Lung_Opacity-5703.png,True,False,299,299,L,116.29,54d23d660b37f7cc703166d1534e643f
Lung_Opacity-5704.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5704.png,Lung_Opacity/images/Lung_Opacity-5704.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5704.png,Lung_Opacity/masks/Lung_Opacity-5704.png,True,False,299,299,L,111.63,defb60a6f7047cb225f1f5a43d598db7
Lung_Opacity-5705.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5705.png,Lung_Opacity/images/Lung_Opacity-5705.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5705.png,Lung_Opacity/masks/Lung_Opacity-5705.png,True,False,299,299,L,104.79,7cf08f8c66cfbd846f0b470bdce4a495
Lung_Opacity-5706.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5706.png,Lung_Opacity/images/Lung_Opacity-5706.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5706.png,Lung_Opacity/masks/Lung_Opacity-5706.png,True,False,299,299,L,156.67,c8a30845967c82dcc84bbf7f93dce080
Lung_Opacity-5707.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5707.png,Lung_Opacity/images/Lung_Opacity-5707.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5707.png,Lung_Opacity/masks/Lung_Opacity-5707.png,True,False,299,299,L,146.34,e62ce1cfba5a5a9c9d504d3555b372a7
Lung_Opacity-5708.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5708.png,Lung_Opacity/images/Lung_Opacity-5708.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5708.png,Lung_Opacity/masks/Lung_Opacity-5708.png,True,False,299,299,L,124.44,e446ea3caec59bb3dcdeda7a1700b80c
Lung_Opacity-5709.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5709.png,Lung_Opacity/images/Lung_Opacity-5709.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5709.png,Lung_Opacity/masks/Lung_Opacity-5709.png,True,False,299,299,L,112.36,c7c7bad077e6d14a732cef1bffc96a62
Lung_Opacity-571.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-571.png,Lung_Opacity/images/Lung_Opacity-571.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-571.png,Lung_Opacity/masks/Lung_Opacity-571.png,True,False,299,299,L,115.24,1a5dec7ec1417eaaaba6e9a1e2701f14
Lung_Opacity-5710.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5710.png,Lung_Opacity/images/Lung_Opacity-5710.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5710.png,Lung_Opacity/masks/Lung_Opacity-5710.png,True,False,299,299,L,127.8,bc2ff6d70e7f0a03f2c7354f792524d5
Lung_Opacity-5711.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5711.png,Lung_Opacity/images/Lung_Opacity-5711.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5711.png,Lung_Opacity/masks/Lung_Opacity-5711.png,True,False,299,299,L,149.79,0f3683a86a7c2e7031da9b57435bc8fa
Lung_Opacity-5712.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5712.png,Lung_Opacity/images/Lung_Opacity-5712.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5712.png,Lung_Opacity/masks/Lung_Opacity-5712.png,True,False,299,299,L,118.87,25cc48c5e3f6742a3ca5c8c17ed20933
Lung_Opacity-5713.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5713.png,Lung_Opacity/images/Lung_Opacity-5713.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5713.png,Lung_Opacity/masks/Lung_Opacity-5713.png,True,False,299,299,L,154.44,b5e7b8da5542217508e1fb272ac21b67
Lung_Opacity-5714.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5714.png,Lung_Opacity/images/Lung_Opacity-5714.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5714.png,Lung_Opacity/masks/Lung_Opacity-5714.png,True,False,299,299,L,109.74,c1158dfa4ab9ccba45497ef8d26ceec8
Lung_Opacity-5715.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5715.png,Lung_Opacity/images/Lung_Opacity-5715.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5715.png,Lung_Opacity/masks/Lung_Opacity-5715.png,True,False,299,299,L,115.64,2be7ad18b386942b7be2d1cd7ac53e18
Lung_Opacity-5716.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5716.png,Lung_Opacity/images/Lung_Opacity-5716.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5716.png,Lung_Opacity/masks/Lung_Opacity-5716.png,True,False,299,299,L,102.1,ec3ac202701ad1cb0597d71105afed8a
Lung_Opacity-5717.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5717.png,Lung_Opacity/images/Lung_Opacity-5717.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5717.png,Lung_Opacity/masks/Lung_Opacity-5717.png,True,False,299,299,L,109.48,18c5d2bc7578dda87b9973081084584a
Lung_Opacity-5718.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5718.png,Lung_Opacity/images/Lung_Opacity-5718.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5718.png,Lung_Opacity/masks/Lung_Opacity-5718.png,True,False,299,299,L,113.96,29004d3ce6dc5cf421a549243ea38fba
Lung_Opacity-5719.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5719.png,Lung_Opacity/images/Lung_Opacity-5719.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5719.png,Lung_Opacity/masks/Lung_Opacity-5719.png,True,False,299,299,L,139.39,27aecab5ec2c9b13e3dddae32f2c528c
Lung_Opacity-572.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-572.png,Lung_Opacity/images/Lung_Opacity-572.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-572.png,Lung_Opacity/masks/Lung_Opacity-572.png,True,False,299,299,L,149.3,099d5ca1cc49f0dfcd69471a496987bc
Lung_Opacity-5720.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5720.png,Lung_Opacity/images/Lung_Opacity-5720.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5720.png,Lung_Opacity/masks/Lung_Opacity-5720.png,True,False,299,299,L,124.16,d95ec440af4cf1516efe1cbd57bfa528
Lung_Opacity-5721.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5721.png,Lung_Opacity/images/Lung_Opacity-5721.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5721.png,Lung_Opacity/masks/Lung_Opacity-5721.png,True,False,299,299,L,108.91,4ceed87d74f046e10dd20a37149a90b2
Lung_Opacity-5722.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5722.png,Lung_Opacity/images/Lung_Opacity-5722.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5722.png,Lung_Opacity/masks/Lung_Opacity-5722.png,True,False,299,299,L,125.99,013fca03556489731a70f48a79a59732
Lung_Opacity-5723.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5723.png,Lung_Opacity/images/Lung_Opacity-5723.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5723.png,Lung_Opacity/masks/Lung_Opacity-5723.png,True,False,299,299,L,99.48,cd2fc6d58b647cdbe12d78286bda83b4
Lung_Opacity-5724.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5724.png,Lung_Opacity/images/Lung_Opacity-5724.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5724.png,Lung_Opacity/masks/Lung_Opacity-5724.png,True,False,299,299,L,111.71,13740ddb3de52b634e7e4ed3385bf4f5
Lung_Opacity-5725.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5725.png,Lung_Opacity/images/Lung_Opacity-5725.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5725.png,Lung_Opacity/masks/Lung_Opacity-5725.png,True,False,299,299,L,100.1,a88901a97889af4101b78e3c5bff749a
Lung_Opacity-5726.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5726.png,Lung_Opacity/images/Lung_Opacity-5726.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5726.png,Lung_Opacity/masks/Lung_Opacity-5726.png,True,False,299,299,L,95.7,af7c91e7256c12bd4bbfcca14e07ab88
Lung_Opacity-5727.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5727.png,Lung_Opacity/images/Lung_Opacity-5727.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5727.png,Lung_Opacity/masks/Lung_Opacity-5727.png,True,False,299,299,L,96.32,f0f2702a0a8cb2c00bd7328548320ab7
Lung_Opacity-5728.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5728.png,Lung_Opacity/images/Lung_Opacity-5728.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5728.png,Lung_Opacity/masks/Lung_Opacity-5728.png,True,False,299,299,L,119.66,c2d2919d31305fcf644381591929f2b7
Lung_Opacity-5729.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5729.png,Lung_Opacity/images/Lung_Opacity-5729.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5729.png,Lung_Opacity/masks/Lung_Opacity-5729.png,True,False,299,299,L,109.31,cb69fc7dea87527cf4328c8c5bbaf273
Lung_Opacity-573.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-573.png,Lung_Opacity/images/Lung_Opacity-573.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-573.png,Lung_Opacity/masks/Lung_Opacity-573.png,True,False,299,299,L,111.2,825f4dc763da82db6d42457a5d0dce98
Lung_Opacity-5730.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5730.png,Lung_Opacity/images/Lung_Opacity-5730.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5730.png,Lung_Opacity/masks/Lung_Opacity-5730.png,True,False,299,299,L,137.58,1ee022e2c2d669f91c2e8ee25748016c
Lung_Opacity-5731.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5731.png,Lung_Opacity/images/Lung_Opacity-5731.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5731.png,Lung_Opacity/masks/Lung_Opacity-5731.png,True,False,299,299,L,120.98,a2cb1c45ae49a27991f344132eff436b
Lung_Opacity-5732.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5732.png,Lung_Opacity/images/Lung_Opacity-5732.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5732.png,Lung_Opacity/masks/Lung_Opacity-5732.png,True,False,299,299,L,127.46,d57bbb2a003edcddd9dc218480d159dc
Lung_Opacity-5733.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5733.png,Lung_Opacity/images/Lung_Opacity-5733.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5733.png,Lung_Opacity/masks/Lung_Opacity-5733.png,True,False,299,299,L,113.87,dbcc4ec88ec109c1390f1bc87be2dd1c
Lung_Opacity-5734.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5734.png,Lung_Opacity/images/Lung_Opacity-5734.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5734.png,Lung_Opacity/masks/Lung_Opacity-5734.png,True,False,299,299,L,161.71,6e2dad8218885dc0de80e69d1c9763af
Lung_Opacity-5735.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5735.png,Lung_Opacity/images/Lung_Opacity-5735.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5735.png,Lung_Opacity/masks/Lung_Opacity-5735.png,True,False,299,299,L,137.29,026994fb73a424864cdcb3a3cfe3750d
Lung_Opacity-5736.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5736.png,Lung_Opacity/images/Lung_Opacity-5736.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5736.png,Lung_Opacity/masks/Lung_Opacity-5736.png,True,False,299,299,L,133.28,5bba83edbc7409cbdd4c0a987089e203
Lung_Opacity-5737.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5737.png,Lung_Opacity/images/Lung_Opacity-5737.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5737.png,Lung_Opacity/masks/Lung_Opacity-5737.png,True,False,299,299,L,93.83,3496380544436926596bf7d0edb74fcf
Lung_Opacity-5738.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5738.png,Lung_Opacity/images/Lung_Opacity-5738.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5738.png,Lung_Opacity/masks/Lung_Opacity-5738.png,True,False,299,299,L,111.44,5ce7ddbfa0fc36f3d15de8537e991a26
Lung_Opacity-5739.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5739.png,Lung_Opacity/images/Lung_Opacity-5739.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5739.png,Lung_Opacity/masks/Lung_Opacity-5739.png,True,False,299,299,L,121.35,74762d05c30b6ff00e193003a8200204
Lung_Opacity-574.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-574.png,Lung_Opacity/images/Lung_Opacity-574.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-574.png,Lung_Opacity/masks/Lung_Opacity-574.png,True,False,299,299,L,108.75,b1298edff9cc00b5e497a01626b4d9b8
Lung_Opacity-5740.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5740.png,Lung_Opacity/images/Lung_Opacity-5740.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5740.png,Lung_Opacity/masks/Lung_Opacity-5740.png,True,False,299,299,L,102.54,d416993304c2be6583003c5121b9a553
Lung_Opacity-5741.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5741.png,Lung_Opacity/images/Lung_Opacity-5741.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5741.png,Lung_Opacity/masks/Lung_Opacity-5741.png,True,False,299,299,L,133.18,c6f66304762a8ed88e1a51b3d87d2d1d
Lung_Opacity-5742.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5742.png,Lung_Opacity/images/Lung_Opacity-5742.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5742.png,Lung_Opacity/masks/Lung_Opacity-5742.png,True,False,299,299,L,121.41,5a3f9def212ccd17df945e6acc124336
Lung_Opacity-5743.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5743.png,Lung_Opacity/images/Lung_Opacity-5743.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5743.png,Lung_Opacity/masks/Lung_Opacity-5743.png,True,False,299,299,L,113.66,c0c39aaeea33f822fcbd030df4dd4290
Lung_Opacity-5744.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5744.png,Lung_Opacity/images/Lung_Opacity-5744.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5744.png,Lung_Opacity/masks/Lung_Opacity-5744.png,True,False,299,299,L,114.16,3bcb3430caa64e3178b80c32f7666243
Lung_Opacity-5745.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5745.png,Lung_Opacity/images/Lung_Opacity-5745.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5745.png,Lung_Opacity/masks/Lung_Opacity-5745.png,True,False,299,299,L,120.15,9aae26c43e84fc5ad9c38392dd117b2a
Lung_Opacity-5746.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5746.png,Lung_Opacity/images/Lung_Opacity-5746.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5746.png,Lung_Opacity/masks/Lung_Opacity-5746.png,True,False,299,299,L,109.87,84d9d76d9471f20450440894072a88c7
Lung_Opacity-5747.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5747.png,Lung_Opacity/images/Lung_Opacity-5747.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5747.png,Lung_Opacity/masks/Lung_Opacity-5747.png,True,False,299,299,L,154.37,6116b61ada275e3cf6a57d12d03eaec8
Lung_Opacity-5748.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5748.png,Lung_Opacity/images/Lung_Opacity-5748.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5748.png,Lung_Opacity/masks/Lung_Opacity-5748.png,True,False,299,299,L,133.13,cd525aade6055191efd2a55b9c1782e1
Lung_Opacity-5749.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5749.png,Lung_Opacity/images/Lung_Opacity-5749.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5749.png,Lung_Opacity/masks/Lung_Opacity-5749.png,True,False,299,299,L,131.69,0d0968b018df34aba794029426e14340
Lung_Opacity-575.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-575.png,Lung_Opacity/images/Lung_Opacity-575.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-575.png,Lung_Opacity/masks/Lung_Opacity-575.png,True,False,299,299,L,150.44,3a262364d63915c9fbb5702ddc0e90d3
Lung_Opacity-5750.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5750.png,Lung_Opacity/images/Lung_Opacity-5750.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5750.png,Lung_Opacity/masks/Lung_Opacity-5750.png,True,False,299,299,L,130.27,4073e143bd5eb3c29c0d536989b9a033
Lung_Opacity-5751.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5751.png,Lung_Opacity/images/Lung_Opacity-5751.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5751.png,Lung_Opacity/masks/Lung_Opacity-5751.png,True,False,299,299,L,152.68,ca3cbc3091a54ddcae7cf91ccaae690f
Lung_Opacity-5752.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5752.png,Lung_Opacity/images/Lung_Opacity-5752.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5752.png,Lung_Opacity/masks/Lung_Opacity-5752.png,True,False,299,299,L,177.05,65b850ee71b97e69b6a67882d6f3a167
Lung_Opacity-5753.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5753.png,Lung_Opacity/images/Lung_Opacity-5753.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5753.png,Lung_Opacity/masks/Lung_Opacity-5753.png,True,False,299,299,L,168.71,9fea2dea016531fd89d6fca389070c11
Lung_Opacity-5754.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5754.png,Lung_Opacity/images/Lung_Opacity-5754.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5754.png,Lung_Opacity/masks/Lung_Opacity-5754.png,True,False,299,299,L,120.65,2cd1e5df6c3265036be1957421116f8d
Lung_Opacity-5755.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5755.png,Lung_Opacity/images/Lung_Opacity-5755.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5755.png,Lung_Opacity/masks/Lung_Opacity-5755.png,True,False,299,299,L,154.59,61c6c4966d3b6ccb013ea8f0e4809545
Lung_Opacity-5756.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5756.png,Lung_Opacity/images/Lung_Opacity-5756.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5756.png,Lung_Opacity/masks/Lung_Opacity-5756.png,True,False,299,299,L,118.37,698f4f93e054738af89d26afe9c00dea
Lung_Opacity-5757.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5757.png,Lung_Opacity/images/Lung_Opacity-5757.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5757.png,Lung_Opacity/masks/Lung_Opacity-5757.png,True,False,299,299,L,131.42,009ad557c19b646f7a78d65fa6034aea
Lung_Opacity-5758.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5758.png,Lung_Opacity/images/Lung_Opacity-5758.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5758.png,Lung_Opacity/masks/Lung_Opacity-5758.png,True,False,299,299,L,121.02,092f1862c72f9cb2dbf7aa6df69b7ad2
Lung_Opacity-5759.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5759.png,Lung_Opacity/images/Lung_Opacity-5759.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5759.png,Lung_Opacity/masks/Lung_Opacity-5759.png,True,False,299,299,L,158.54,ff672879bede793d179fff2feba75f65
Lung_Opacity-576.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-576.png,Lung_Opacity/images/Lung_Opacity-576.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-576.png,Lung_Opacity/masks/Lung_Opacity-576.png,True,False,299,299,L,139.75,4601a983da869779b30a8ab2d29e2157
Lung_Opacity-5760.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5760.png,Lung_Opacity/images/Lung_Opacity-5760.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5760.png,Lung_Opacity/masks/Lung_Opacity-5760.png,True,False,299,299,L,115.52,ba3279782a45d938366e7a1903b39d5a
Lung_Opacity-5761.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5761.png,Lung_Opacity/images/Lung_Opacity-5761.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5761.png,Lung_Opacity/masks/Lung_Opacity-5761.png,True,False,299,299,L,152.53,538cb69db255c0bf65a2994257cbcf49
Lung_Opacity-5762.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5762.png,Lung_Opacity/images/Lung_Opacity-5762.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5762.png,Lung_Opacity/masks/Lung_Opacity-5762.png,True,False,299,299,L,113.3,280de704d163a0878136000a2168be10
Lung_Opacity-5763.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5763.png,Lung_Opacity/images/Lung_Opacity-5763.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5763.png,Lung_Opacity/masks/Lung_Opacity-5763.png,True,False,299,299,L,101.58,81d1f5abac2724cc273ae893f5885b58
Lung_Opacity-5764.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5764.png,Lung_Opacity/images/Lung_Opacity-5764.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5764.png,Lung_Opacity/masks/Lung_Opacity-5764.png,True,False,299,299,L,102.45,8cdd1815ed47646889c3802e9f8f3c58
Lung_Opacity-5765.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5765.png,Lung_Opacity/images/Lung_Opacity-5765.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5765.png,Lung_Opacity/masks/Lung_Opacity-5765.png,True,False,299,299,L,118.73,e76b6e3c68b8046b9fdc36074318b6d5
Lung_Opacity-5766.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5766.png,Lung_Opacity/images/Lung_Opacity-5766.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5766.png,Lung_Opacity/masks/Lung_Opacity-5766.png,True,False,299,299,L,130.86,7ff57c53a7bc8f40d133b457684da306
Lung_Opacity-5767.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5767.png,Lung_Opacity/images/Lung_Opacity-5767.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5767.png,Lung_Opacity/masks/Lung_Opacity-5767.png,True,False,299,299,L,161.71,b2921e66de469fde376e022b19531555
Lung_Opacity-5768.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5768.png,Lung_Opacity/images/Lung_Opacity-5768.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5768.png,Lung_Opacity/masks/Lung_Opacity-5768.png,True,False,299,299,L,101.13,f8934582d7578d28260a3cefb8c45e32
Lung_Opacity-5769.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5769.png,Lung_Opacity/images/Lung_Opacity-5769.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5769.png,Lung_Opacity/masks/Lung_Opacity-5769.png,True,False,299,299,L,113.63,ef7ac9c3236c3c4caf87c6a7708b0c84
Lung_Opacity-577.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-577.png,Lung_Opacity/images/Lung_Opacity-577.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-577.png,Lung_Opacity/masks/Lung_Opacity-577.png,True,False,299,299,L,118.28,a49b5dddd01ca4358f0e95b1585f3aba
Lung_Opacity-5770.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5770.png,Lung_Opacity/images/Lung_Opacity-5770.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5770.png,Lung_Opacity/masks/Lung_Opacity-5770.png,True,False,299,299,L,128.67,aa2ea2077ed6797f73de73a2b01ac376
Lung_Opacity-5771.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5771.png,Lung_Opacity/images/Lung_Opacity-5771.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5771.png,Lung_Opacity/masks/Lung_Opacity-5771.png,True,False,299,299,L,137.97,cb978c5efd7ad738ab01b99625064306
Lung_Opacity-5772.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5772.png,Lung_Opacity/images/Lung_Opacity-5772.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5772.png,Lung_Opacity/masks/Lung_Opacity-5772.png,True,False,299,299,L,114.88,906f839b24978ae2242c8b1731098fcd
Lung_Opacity-5773.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5773.png,Lung_Opacity/images/Lung_Opacity-5773.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5773.png,Lung_Opacity/masks/Lung_Opacity-5773.png,True,False,299,299,L,108.89,e335461143b22994ed28081a536a7cdd
Lung_Opacity-5774.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5774.png,Lung_Opacity/images/Lung_Opacity-5774.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5774.png,Lung_Opacity/masks/Lung_Opacity-5774.png,True,False,299,299,L,117.81,9215f6d3022b14636d697d7e2d11b3cb
Lung_Opacity-5775.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5775.png,Lung_Opacity/images/Lung_Opacity-5775.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5775.png,Lung_Opacity/masks/Lung_Opacity-5775.png,True,False,299,299,L,141.22,20974d2a680604749c549ff818edc1b1
Lung_Opacity-5776.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5776.png,Lung_Opacity/images/Lung_Opacity-5776.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5776.png,Lung_Opacity/masks/Lung_Opacity-5776.png,True,False,299,299,L,105.58,84ad20f4c78c4fa240981bff1922fc56
Lung_Opacity-5777.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5777.png,Lung_Opacity/images/Lung_Opacity-5777.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5777.png,Lung_Opacity/masks/Lung_Opacity-5777.png,True,False,299,299,L,125.25,e6e2efb3f0b43e716c2f13584a6c3fa6
Lung_Opacity-5778.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5778.png,Lung_Opacity/images/Lung_Opacity-5778.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5778.png,Lung_Opacity/masks/Lung_Opacity-5778.png,True,False,299,299,L,112.84,df2aac1a49c40579e12826d97a3cde88
Lung_Opacity-5779.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5779.png,Lung_Opacity/images/Lung_Opacity-5779.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5779.png,Lung_Opacity/masks/Lung_Opacity-5779.png,True,False,299,299,L,177.97,38f5679d3aaaf36357c8c1e9815fe9e3
Lung_Opacity-578.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-578.png,Lung_Opacity/images/Lung_Opacity-578.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-578.png,Lung_Opacity/masks/Lung_Opacity-578.png,True,False,299,299,L,110.04,67eaee1b54d394759375005fe9ef1b64
Lung_Opacity-5780.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5780.png,Lung_Opacity/images/Lung_Opacity-5780.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5780.png,Lung_Opacity/masks/Lung_Opacity-5780.png,True,False,299,299,L,132.74,7e06497b14ba276f30d0e8b4b5604faf
Lung_Opacity-5781.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5781.png,Lung_Opacity/images/Lung_Opacity-5781.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5781.png,Lung_Opacity/masks/Lung_Opacity-5781.png,True,False,299,299,L,109.89,0e916e71d79058e8c67f78e1fb3c6ee9
Lung_Opacity-5782.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5782.png,Lung_Opacity/images/Lung_Opacity-5782.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5782.png,Lung_Opacity/masks/Lung_Opacity-5782.png,True,False,299,299,L,120.01,2f90d356680e7c25c13913d2409e46f9
Lung_Opacity-5783.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5783.png,Lung_Opacity/images/Lung_Opacity-5783.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5783.png,Lung_Opacity/masks/Lung_Opacity-5783.png,True,False,299,299,L,112.83,83cad10a26554be3c29980c7f4c44b79
Lung_Opacity-5784.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5784.png,Lung_Opacity/images/Lung_Opacity-5784.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5784.png,Lung_Opacity/masks/Lung_Opacity-5784.png,True,False,299,299,L,78.69,d5bed79c51249627b2825aac08ce06f3
Lung_Opacity-5785.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5785.png,Lung_Opacity/images/Lung_Opacity-5785.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5785.png,Lung_Opacity/masks/Lung_Opacity-5785.png,True,False,299,299,L,108.72,6185d34b9df2eb7b70ecdf1e0b6deafc
Lung_Opacity-5786.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5786.png,Lung_Opacity/images/Lung_Opacity-5786.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5786.png,Lung_Opacity/masks/Lung_Opacity-5786.png,True,False,299,299,L,158.15,beb2f07d706a6e8bf258b51017419dab
Lung_Opacity-5787.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5787.png,Lung_Opacity/images/Lung_Opacity-5787.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5787.png,Lung_Opacity/masks/Lung_Opacity-5787.png,True,False,299,299,L,126.68,bfa3c91f838a39bb93b3ae28032f073b
Lung_Opacity-5788.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5788.png,Lung_Opacity/images/Lung_Opacity-5788.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5788.png,Lung_Opacity/masks/Lung_Opacity-5788.png,True,False,299,299,L,148.07,3ffe4c00dd55b8c45e43c23e824daf92
Lung_Opacity-5789.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5789.png,Lung_Opacity/images/Lung_Opacity-5789.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5789.png,Lung_Opacity/masks/Lung_Opacity-5789.png,True,False,299,299,L,119.01,c6b9655a97daac76027bc2bf29561823
Lung_Opacity-579.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-579.png,Lung_Opacity/images/Lung_Opacity-579.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-579.png,Lung_Opacity/masks/Lung_Opacity-579.png,True,False,299,299,L,95.57,ecc6915ae40759e318cc5939507c7afe
Lung_Opacity-5790.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5790.png,Lung_Opacity/images/Lung_Opacity-5790.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5790.png,Lung_Opacity/masks/Lung_Opacity-5790.png,True,False,299,299,L,94.38,8b4691b91fdfaca13a98ca02304c7652
Lung_Opacity-5791.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5791.png,Lung_Opacity/images/Lung_Opacity-5791.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5791.png,Lung_Opacity/masks/Lung_Opacity-5791.png,True,False,299,299,L,113.05,8f4ac4111d636f1352312e96bdc663c0
Lung_Opacity-5792.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5792.png,Lung_Opacity/images/Lung_Opacity-5792.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5792.png,Lung_Opacity/masks/Lung_Opacity-5792.png,True,False,299,299,L,123.57,5de320fc52f128f37d42a4d6b6f07b13
Lung_Opacity-5793.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5793.png,Lung_Opacity/images/Lung_Opacity-5793.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5793.png,Lung_Opacity/masks/Lung_Opacity-5793.png,True,False,299,299,L,156.32,f3537383ad7431ce22049e1d4bc93b9d
Lung_Opacity-5794.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5794.png,Lung_Opacity/images/Lung_Opacity-5794.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5794.png,Lung_Opacity/masks/Lung_Opacity-5794.png,True,False,299,299,L,129.71,d83668e4404a40eaca88feffaf706d00
Lung_Opacity-5795.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5795.png,Lung_Opacity/images/Lung_Opacity-5795.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5795.png,Lung_Opacity/masks/Lung_Opacity-5795.png,True,False,299,299,L,86.0,ddfc860bfa6cf96e744ffe2be43e156d
Lung_Opacity-5796.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5796.png,Lung_Opacity/images/Lung_Opacity-5796.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5796.png,Lung_Opacity/masks/Lung_Opacity-5796.png,True,False,299,299,L,118.55,f98ba966dc4e72f678ec4a87f2ec957f
Lung_Opacity-5797.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5797.png,Lung_Opacity/images/Lung_Opacity-5797.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5797.png,Lung_Opacity/masks/Lung_Opacity-5797.png,True,False,299,299,L,151.04,ff683d4c534c09c014544a3f6b4becea
Lung_Opacity-5798.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5798.png,Lung_Opacity/images/Lung_Opacity-5798.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5798.png,Lung_Opacity/masks/Lung_Opacity-5798.png,True,False,299,299,L,148.11,7c3ac46d78cef2efd89a5eeea9fc7a4b
Lung_Opacity-5799.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5799.png,Lung_Opacity/images/Lung_Opacity-5799.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5799.png,Lung_Opacity/masks/Lung_Opacity-5799.png,True,False,299,299,L,117.62,4f529658267a9644c8ee27fc4246f707
Lung_Opacity-58.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-58.png,Lung_Opacity/images/Lung_Opacity-58.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-58.png,Lung_Opacity/masks/Lung_Opacity-58.png,True,False,299,299,L,95.97,a3d225d2c88c5c7c78c38ba743606d6e
Lung_Opacity-580.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-580.png,Lung_Opacity/images/Lung_Opacity-580.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-580.png,Lung_Opacity/masks/Lung_Opacity-580.png,True,False,299,299,L,174.74,8edb9dd1e87b78f944f45404676af7b3
Lung_Opacity-5800.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5800.png,Lung_Opacity/images/Lung_Opacity-5800.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5800.png,Lung_Opacity/masks/Lung_Opacity-5800.png,True,False,299,299,L,105.33,1e4a61f885d19e29cebf7a8c8309068c
Lung_Opacity-5801.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5801.png,Lung_Opacity/images/Lung_Opacity-5801.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5801.png,Lung_Opacity/masks/Lung_Opacity-5801.png,True,False,299,299,L,107.83,59aefefaf6c0c00a1e3f77e2288e018c
Lung_Opacity-5802.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5802.png,Lung_Opacity/images/Lung_Opacity-5802.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5802.png,Lung_Opacity/masks/Lung_Opacity-5802.png,True,False,299,299,L,169.21,15b74cd57cbf3e19176542b923cbf8dc
Lung_Opacity-5803.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5803.png,Lung_Opacity/images/Lung_Opacity-5803.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5803.png,Lung_Opacity/masks/Lung_Opacity-5803.png,True,False,299,299,L,108.15,0cf2d2d7ec00b986b4cd2cc1b8511c52
Lung_Opacity-5804.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5804.png,Lung_Opacity/images/Lung_Opacity-5804.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5804.png,Lung_Opacity/masks/Lung_Opacity-5804.png,True,False,299,299,L,142.24,2f8cfd931ec95dc39ca657deadcb0149
Lung_Opacity-5805.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5805.png,Lung_Opacity/images/Lung_Opacity-5805.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5805.png,Lung_Opacity/masks/Lung_Opacity-5805.png,True,False,299,299,L,110.13,b14eb3a92df05c05c4053fc295ac5352
Lung_Opacity-5806.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5806.png,Lung_Opacity/images/Lung_Opacity-5806.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5806.png,Lung_Opacity/masks/Lung_Opacity-5806.png,True,False,299,299,L,94.4,13a39c0e54a66ff8d0bd59d675f27e15
Lung_Opacity-5807.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5807.png,Lung_Opacity/images/Lung_Opacity-5807.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5807.png,Lung_Opacity/masks/Lung_Opacity-5807.png,True,False,299,299,L,162.91,e537413fae1d53b87cb8a7aadf7d99d7
Lung_Opacity-5808.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5808.png,Lung_Opacity/images/Lung_Opacity-5808.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5808.png,Lung_Opacity/masks/Lung_Opacity-5808.png,True,False,299,299,L,92.28,6dfc2fdb5199791f865a34b3f1d1c44a
Lung_Opacity-5809.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5809.png,Lung_Opacity/images/Lung_Opacity-5809.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5809.png,Lung_Opacity/masks/Lung_Opacity-5809.png,True,False,299,299,L,168.85,a832cfc261006677cb8fa156cec3ffdd
Lung_Opacity-581.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-581.png,Lung_Opacity/images/Lung_Opacity-581.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-581.png,Lung_Opacity/masks/Lung_Opacity-581.png,True,False,299,299,L,110.21,c104356bc194e34faaa71be226dd336a
Lung_Opacity-5810.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5810.png,Lung_Opacity/images/Lung_Opacity-5810.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5810.png,Lung_Opacity/masks/Lung_Opacity-5810.png,True,False,299,299,L,119.34,a799f2118b89e0ca4670b8bbd282340c
Lung_Opacity-5811.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5811.png,Lung_Opacity/images/Lung_Opacity-5811.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5811.png,Lung_Opacity/masks/Lung_Opacity-5811.png,True,False,299,299,L,113.76,b350527c4848f82667273c2cd13bd075
Lung_Opacity-5812.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5812.png,Lung_Opacity/images/Lung_Opacity-5812.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5812.png,Lung_Opacity/masks/Lung_Opacity-5812.png,True,False,299,299,L,116.52,2c800b75390f90c5abee45fd50bb4433
Lung_Opacity-5813.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5813.png,Lung_Opacity/images/Lung_Opacity-5813.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5813.png,Lung_Opacity/masks/Lung_Opacity-5813.png,True,False,299,299,L,88.41,67b25c9f9033436e1d1ebadbe9a74d7f
Lung_Opacity-5814.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5814.png,Lung_Opacity/images/Lung_Opacity-5814.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5814.png,Lung_Opacity/masks/Lung_Opacity-5814.png,True,False,299,299,L,165.61,4a1f45ffebf7c4fbea2f95446ab5e12a
Lung_Opacity-5815.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5815.png,Lung_Opacity/images/Lung_Opacity-5815.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5815.png,Lung_Opacity/masks/Lung_Opacity-5815.png,True,False,299,299,L,136.64,1cfcc50b53809791645905abd0396791
Lung_Opacity-5816.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5816.png,Lung_Opacity/images/Lung_Opacity-5816.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5816.png,Lung_Opacity/masks/Lung_Opacity-5816.png,True,False,299,299,L,143.79,f77ff5c6e672dd979e1b247670db771a
Lung_Opacity-5817.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5817.png,Lung_Opacity/images/Lung_Opacity-5817.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5817.png,Lung_Opacity/masks/Lung_Opacity-5817.png,True,False,299,299,L,146.96,099e59988f96db8a01002f54684069ec
Lung_Opacity-5818.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5818.png,Lung_Opacity/images/Lung_Opacity-5818.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5818.png,Lung_Opacity/masks/Lung_Opacity-5818.png,True,False,299,299,L,148.39,ec386ea9fff40e292a97d46cc9ce4417
Lung_Opacity-5819.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5819.png,Lung_Opacity/images/Lung_Opacity-5819.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5819.png,Lung_Opacity/masks/Lung_Opacity-5819.png,True,False,299,299,L,146.19,97d11fd0f0af980ea782c9baf9b0f690
Lung_Opacity-582.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-582.png,Lung_Opacity/images/Lung_Opacity-582.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-582.png,Lung_Opacity/masks/Lung_Opacity-582.png,True,False,299,299,L,165.19,d5a085eba3141a9e949dd5e4d6ea01ae
Lung_Opacity-5820.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5820.png,Lung_Opacity/images/Lung_Opacity-5820.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5820.png,Lung_Opacity/masks/Lung_Opacity-5820.png,True,False,299,299,L,112.85,d469ddfef9cd0409bbe3a959fbbd07dc
Lung_Opacity-5821.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5821.png,Lung_Opacity/images/Lung_Opacity-5821.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5821.png,Lung_Opacity/masks/Lung_Opacity-5821.png,True,False,299,299,L,122.86,a2089ae5cf7417fbf8dab2b8dc2c8abe
Lung_Opacity-5822.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5822.png,Lung_Opacity/images/Lung_Opacity-5822.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5822.png,Lung_Opacity/masks/Lung_Opacity-5822.png,True,False,299,299,L,136.91,fb702025b8578f7d5962aae55b6f32ae
Lung_Opacity-5823.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5823.png,Lung_Opacity/images/Lung_Opacity-5823.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5823.png,Lung_Opacity/masks/Lung_Opacity-5823.png,True,False,299,299,L,111.29,8bf7deb270fd21eada216cd57a3b7bda
Lung_Opacity-5824.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5824.png,Lung_Opacity/images/Lung_Opacity-5824.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5824.png,Lung_Opacity/masks/Lung_Opacity-5824.png,True,False,299,299,L,163.17,05da1366465b4be7cfab092b0478300b
Lung_Opacity-5825.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5825.png,Lung_Opacity/images/Lung_Opacity-5825.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5825.png,Lung_Opacity/masks/Lung_Opacity-5825.png,True,False,299,299,L,128.6,7dd9cacef934291f374f6af2f540f141
Lung_Opacity-5826.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5826.png,Lung_Opacity/images/Lung_Opacity-5826.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5826.png,Lung_Opacity/masks/Lung_Opacity-5826.png,True,False,299,299,L,116.2,70798821cf9df9e3ead913f3de349da7
Lung_Opacity-5827.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5827.png,Lung_Opacity/images/Lung_Opacity-5827.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5827.png,Lung_Opacity/masks/Lung_Opacity-5827.png,True,False,299,299,L,153.16,84459cf1599a45f683875e68519407ef
Lung_Opacity-5828.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5828.png,Lung_Opacity/images/Lung_Opacity-5828.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5828.png,Lung_Opacity/masks/Lung_Opacity-5828.png,True,False,299,299,L,168.16,84b8ab8019590caef5c04f86756f7ece
Lung_Opacity-5829.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5829.png,Lung_Opacity/images/Lung_Opacity-5829.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5829.png,Lung_Opacity/masks/Lung_Opacity-5829.png,True,False,299,299,L,161.21,3c7f86818a1944997670a635403892d5
Lung_Opacity-583.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-583.png,Lung_Opacity/images/Lung_Opacity-583.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-583.png,Lung_Opacity/masks/Lung_Opacity-583.png,True,False,299,299,L,117.82,52cbccdc01d091aeeeec43c2d08e1bd0
Lung_Opacity-5830.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5830.png,Lung_Opacity/images/Lung_Opacity-5830.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5830.png,Lung_Opacity/masks/Lung_Opacity-5830.png,True,False,299,299,L,136.43,9657ea8bf6594afe501fe3fb4c15ef76
Lung_Opacity-5831.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5831.png,Lung_Opacity/images/Lung_Opacity-5831.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5831.png,Lung_Opacity/masks/Lung_Opacity-5831.png,True,False,299,299,L,163.49,d9ede625d38c21681eeccc38eed5dc1c
Lung_Opacity-5832.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5832.png,Lung_Opacity/images/Lung_Opacity-5832.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5832.png,Lung_Opacity/masks/Lung_Opacity-5832.png,True,False,299,299,L,142.14,614ede0ff82933e80d59a3da1aecd525
Lung_Opacity-5833.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5833.png,Lung_Opacity/images/Lung_Opacity-5833.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5833.png,Lung_Opacity/masks/Lung_Opacity-5833.png,True,False,299,299,L,129.46,355373da2ebc9b0351f3f806199d4081
Lung_Opacity-5834.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5834.png,Lung_Opacity/images/Lung_Opacity-5834.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5834.png,Lung_Opacity/masks/Lung_Opacity-5834.png,True,False,299,299,L,115.71,dd09cb342abe5737493862ac9c736e13
Lung_Opacity-5835.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5835.png,Lung_Opacity/images/Lung_Opacity-5835.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5835.png,Lung_Opacity/masks/Lung_Opacity-5835.png,True,False,299,299,L,93.5,7b1abde42c9df1b93d0c85e35daec5aa
Lung_Opacity-5836.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5836.png,Lung_Opacity/images/Lung_Opacity-5836.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5836.png,Lung_Opacity/masks/Lung_Opacity-5836.png,True,False,299,299,L,151.73,0a2ce73db0ae6c52060f5d97ab3e8d26
Lung_Opacity-5837.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5837.png,Lung_Opacity/images/Lung_Opacity-5837.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5837.png,Lung_Opacity/masks/Lung_Opacity-5837.png,True,False,299,299,L,115.56,a174571933064a97f4d136cdcd8eec6c
Lung_Opacity-5838.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5838.png,Lung_Opacity/images/Lung_Opacity-5838.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5838.png,Lung_Opacity/masks/Lung_Opacity-5838.png,True,False,299,299,L,115.34,7ae23af4c35ca3d3eb3f76a6ed19b87d
Lung_Opacity-5839.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5839.png,Lung_Opacity/images/Lung_Opacity-5839.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5839.png,Lung_Opacity/masks/Lung_Opacity-5839.png,True,False,299,299,L,115.13,6a60577d74b5fbec2838c719b4ed43b8
Lung_Opacity-584.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-584.png,Lung_Opacity/images/Lung_Opacity-584.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-584.png,Lung_Opacity/masks/Lung_Opacity-584.png,True,False,299,299,L,167.12,37a29b7c0bbe62292be05d5a6d4b7284
Lung_Opacity-5840.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5840.png,Lung_Opacity/images/Lung_Opacity-5840.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5840.png,Lung_Opacity/masks/Lung_Opacity-5840.png,True,False,299,299,L,157.71,d2fb06365d366a8ead795d6c86a5e8e3
Lung_Opacity-5841.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5841.png,Lung_Opacity/images/Lung_Opacity-5841.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5841.png,Lung_Opacity/masks/Lung_Opacity-5841.png,True,False,299,299,L,110.62,d98db098c26a105c0bf846519b2799a2
Lung_Opacity-5842.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5842.png,Lung_Opacity/images/Lung_Opacity-5842.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5842.png,Lung_Opacity/masks/Lung_Opacity-5842.png,True,False,299,299,L,106.1,c5143438db1ca041d9673587d262b282
Lung_Opacity-5843.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5843.png,Lung_Opacity/images/Lung_Opacity-5843.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5843.png,Lung_Opacity/masks/Lung_Opacity-5843.png,True,False,299,299,L,123.81,6b1c831433f614cc3ebfdbe809335750
Lung_Opacity-5844.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5844.png,Lung_Opacity/images/Lung_Opacity-5844.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5844.png,Lung_Opacity/masks/Lung_Opacity-5844.png,True,False,299,299,L,135.18,db0e5139d0470b73eb3a9843740e4f61
Lung_Opacity-5845.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5845.png,Lung_Opacity/images/Lung_Opacity-5845.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5845.png,Lung_Opacity/masks/Lung_Opacity-5845.png,True,False,299,299,L,150.97,34fbcda0b7a84926751fa177dd5a2706
Lung_Opacity-5846.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5846.png,Lung_Opacity/images/Lung_Opacity-5846.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5846.png,Lung_Opacity/masks/Lung_Opacity-5846.png,True,False,299,299,L,104.71,d88b178ff625437297fb75a0f59146b4
Lung_Opacity-5847.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5847.png,Lung_Opacity/images/Lung_Opacity-5847.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5847.png,Lung_Opacity/masks/Lung_Opacity-5847.png,True,False,299,299,L,106.57,1c80f65871e7c3877ad23a16df29e8f1
Lung_Opacity-5848.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5848.png,Lung_Opacity/images/Lung_Opacity-5848.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5848.png,Lung_Opacity/masks/Lung_Opacity-5848.png,True,False,299,299,L,117.67,80c0ce6ba8698b74f4e8209b7be31107
Lung_Opacity-5849.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5849.png,Lung_Opacity/images/Lung_Opacity-5849.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5849.png,Lung_Opacity/masks/Lung_Opacity-5849.png,True,False,299,299,L,122.92,6403f66a614db387fe0e8ff6d10ca803
Lung_Opacity-585.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-585.png,Lung_Opacity/images/Lung_Opacity-585.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-585.png,Lung_Opacity/masks/Lung_Opacity-585.png,True,False,299,299,L,124.29,bd2a04d0344b24612bb319e6ee85ea96
Lung_Opacity-5850.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5850.png,Lung_Opacity/images/Lung_Opacity-5850.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5850.png,Lung_Opacity/masks/Lung_Opacity-5850.png,True,False,299,299,L,145.87,8d2ba30991e24ad65fe19eb0ff9dc6d5
Lung_Opacity-5851.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5851.png,Lung_Opacity/images/Lung_Opacity-5851.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5851.png,Lung_Opacity/masks/Lung_Opacity-5851.png,True,False,299,299,L,160.96,3d3bdc8ea92c9e3688883ec0f0aa635a
Lung_Opacity-5852.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5852.png,Lung_Opacity/images/Lung_Opacity-5852.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5852.png,Lung_Opacity/masks/Lung_Opacity-5852.png,True,False,299,299,L,106.26,15d069640c44b7c2fcf19ed0aad47726
Lung_Opacity-5853.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5853.png,Lung_Opacity/images/Lung_Opacity-5853.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5853.png,Lung_Opacity/masks/Lung_Opacity-5853.png,True,False,299,299,L,116.56,323a8c83ba97cef1c77d74810da3dee7
Lung_Opacity-5854.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5854.png,Lung_Opacity/images/Lung_Opacity-5854.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5854.png,Lung_Opacity/masks/Lung_Opacity-5854.png,True,False,299,299,L,109.75,1e48970d292ded05eb673b0b5c81052e
Lung_Opacity-5855.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5855.png,Lung_Opacity/images/Lung_Opacity-5855.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5855.png,Lung_Opacity/masks/Lung_Opacity-5855.png,True,False,299,299,L,123.56,74908abd5933a5eb4c119131c1455f07
Lung_Opacity-5856.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5856.png,Lung_Opacity/images/Lung_Opacity-5856.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5856.png,Lung_Opacity/masks/Lung_Opacity-5856.png,True,False,299,299,L,147.82,7cb6682c6eb8dd2b62c2724c87c1e415
Lung_Opacity-5857.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5857.png,Lung_Opacity/images/Lung_Opacity-5857.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5857.png,Lung_Opacity/masks/Lung_Opacity-5857.png,True,False,299,299,L,131.0,a50cf0058faa7273f5be89dfe7a564c6
Lung_Opacity-5858.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5858.png,Lung_Opacity/images/Lung_Opacity-5858.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5858.png,Lung_Opacity/masks/Lung_Opacity-5858.png,True,False,299,299,L,64.76,dde5b736c3ef70959b51e5d0da181f2e
Lung_Opacity-5859.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5859.png,Lung_Opacity/images/Lung_Opacity-5859.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5859.png,Lung_Opacity/masks/Lung_Opacity-5859.png,True,False,299,299,L,111.38,8812f479dd79bd898d19af45ee6ac235
Lung_Opacity-586.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-586.png,Lung_Opacity/images/Lung_Opacity-586.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-586.png,Lung_Opacity/masks/Lung_Opacity-586.png,True,False,299,299,L,128.5,71291aaf0631ec7d88faad9412c64426
Lung_Opacity-5860.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5860.png,Lung_Opacity/images/Lung_Opacity-5860.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5860.png,Lung_Opacity/masks/Lung_Opacity-5860.png,True,False,299,299,L,90.0,df79e24c2c7aa4bebba41dce91b374e8
Lung_Opacity-5861.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5861.png,Lung_Opacity/images/Lung_Opacity-5861.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5861.png,Lung_Opacity/masks/Lung_Opacity-5861.png,True,False,299,299,L,135.93,12f6a60f31dcf3b65a9e4d10ce851073
Lung_Opacity-5862.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5862.png,Lung_Opacity/images/Lung_Opacity-5862.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5862.png,Lung_Opacity/masks/Lung_Opacity-5862.png,True,False,299,299,L,108.62,59f000cbf3e51f6e291faecebf8c6ffc
Lung_Opacity-5863.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5863.png,Lung_Opacity/images/Lung_Opacity-5863.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5863.png,Lung_Opacity/masks/Lung_Opacity-5863.png,True,False,299,299,L,140.64,bde27ecc32530e2970253d478ec48586
Lung_Opacity-5864.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5864.png,Lung_Opacity/images/Lung_Opacity-5864.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5864.png,Lung_Opacity/masks/Lung_Opacity-5864.png,True,False,299,299,L,129.89,d7e8ad5d25b3727512535156935d6052
Lung_Opacity-5865.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5865.png,Lung_Opacity/images/Lung_Opacity-5865.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5865.png,Lung_Opacity/masks/Lung_Opacity-5865.png,True,False,299,299,L,121.75,268a285e3783352a884a77a77588cf40
Lung_Opacity-5866.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5866.png,Lung_Opacity/images/Lung_Opacity-5866.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5866.png,Lung_Opacity/masks/Lung_Opacity-5866.png,True,False,299,299,L,173.88,5d6b12ad8a0be2f339079d219564a43d
Lung_Opacity-5867.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5867.png,Lung_Opacity/images/Lung_Opacity-5867.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5867.png,Lung_Opacity/masks/Lung_Opacity-5867.png,True,False,299,299,L,86.51,b1eb7d9f198aa3924ba232ca7f4704eb
Lung_Opacity-5868.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5868.png,Lung_Opacity/images/Lung_Opacity-5868.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5868.png,Lung_Opacity/masks/Lung_Opacity-5868.png,True,False,299,299,L,122.94,a2fe5dffd15d6e0f178bd0520c9ff47a
Lung_Opacity-5869.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5869.png,Lung_Opacity/images/Lung_Opacity-5869.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5869.png,Lung_Opacity/masks/Lung_Opacity-5869.png,True,False,299,299,L,123.76,e6790b62500177165a4baa5855ae6f65
Lung_Opacity-587.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-587.png,Lung_Opacity/images/Lung_Opacity-587.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-587.png,Lung_Opacity/masks/Lung_Opacity-587.png,True,False,299,299,L,155.11,a827cc884a7cd67f057372112a73b497
Lung_Opacity-5870.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5870.png,Lung_Opacity/images/Lung_Opacity-5870.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5870.png,Lung_Opacity/masks/Lung_Opacity-5870.png,True,False,299,299,L,106.65,4a27ea0046c9fa272f76e60107417aad
Lung_Opacity-5871.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5871.png,Lung_Opacity/images/Lung_Opacity-5871.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5871.png,Lung_Opacity/masks/Lung_Opacity-5871.png,True,False,299,299,L,148.15,1c48c614ab5eb9c89624da41f0efd4ce
Lung_Opacity-5872.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5872.png,Lung_Opacity/images/Lung_Opacity-5872.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5872.png,Lung_Opacity/masks/Lung_Opacity-5872.png,True,False,299,299,L,116.03,90671a53cbd122e1e5efede2cebce471
Lung_Opacity-5873.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5873.png,Lung_Opacity/images/Lung_Opacity-5873.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5873.png,Lung_Opacity/masks/Lung_Opacity-5873.png,True,False,299,299,L,123.17,2fecd47d48f1e0b3922d4e9612c51787
Lung_Opacity-5874.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5874.png,Lung_Opacity/images/Lung_Opacity-5874.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5874.png,Lung_Opacity/masks/Lung_Opacity-5874.png,True,False,299,299,L,87.18,4277cccf2767c7dc32688b98a6cf42cb
Lung_Opacity-5875.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5875.png,Lung_Opacity/images/Lung_Opacity-5875.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5875.png,Lung_Opacity/masks/Lung_Opacity-5875.png,True,False,299,299,L,115.05,cf7a395a89acf32a8317175c7f0457c9
Lung_Opacity-5876.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5876.png,Lung_Opacity/images/Lung_Opacity-5876.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5876.png,Lung_Opacity/masks/Lung_Opacity-5876.png,True,False,299,299,L,104.82,cc5cd7298486b1124cbcaeedda103447
Lung_Opacity-5877.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5877.png,Lung_Opacity/images/Lung_Opacity-5877.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5877.png,Lung_Opacity/masks/Lung_Opacity-5877.png,True,False,299,299,L,186.68,27f10bc86658f0be5475e44f0bdbd933
Lung_Opacity-5878.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5878.png,Lung_Opacity/images/Lung_Opacity-5878.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5878.png,Lung_Opacity/masks/Lung_Opacity-5878.png,True,False,299,299,L,166.86,df58429ccd897ea4b5e17896ace1b93c
Lung_Opacity-5879.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5879.png,Lung_Opacity/images/Lung_Opacity-5879.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5879.png,Lung_Opacity/masks/Lung_Opacity-5879.png,True,False,299,299,L,98.04,817a78c9065985a551d763d2b12e629d
Lung_Opacity-588.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-588.png,Lung_Opacity/images/Lung_Opacity-588.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-588.png,Lung_Opacity/masks/Lung_Opacity-588.png,True,False,299,299,L,132.97,1b7889aa2fedc054e42b677f9cecfa9c
Lung_Opacity-5880.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5880.png,Lung_Opacity/images/Lung_Opacity-5880.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5880.png,Lung_Opacity/masks/Lung_Opacity-5880.png,True,False,299,299,L,125.76,cb2e1ebf355be0d2321a6ce1d999fa34
Lung_Opacity-5881.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5881.png,Lung_Opacity/images/Lung_Opacity-5881.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5881.png,Lung_Opacity/masks/Lung_Opacity-5881.png,True,False,299,299,L,115.25,f77094d9a0f5506148239d975ecd0b0d
Lung_Opacity-5882.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5882.png,Lung_Opacity/images/Lung_Opacity-5882.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5882.png,Lung_Opacity/masks/Lung_Opacity-5882.png,True,False,299,299,L,114.56,a0ecd5dc0297d7c985e1d994bffb4162
Lung_Opacity-5883.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5883.png,Lung_Opacity/images/Lung_Opacity-5883.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5883.png,Lung_Opacity/masks/Lung_Opacity-5883.png,True,False,299,299,L,122.26,a50238a39906520dcb9830efed75cb19
Lung_Opacity-5884.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5884.png,Lung_Opacity/images/Lung_Opacity-5884.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5884.png,Lung_Opacity/masks/Lung_Opacity-5884.png,True,False,299,299,L,96.07,44581bc99557c7e99500fb8f10680b9c
Lung_Opacity-5885.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5885.png,Lung_Opacity/images/Lung_Opacity-5885.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5885.png,Lung_Opacity/masks/Lung_Opacity-5885.png,True,False,299,299,L,116.03,e102177873602cfa9ba5bba4be130d30
Lung_Opacity-5886.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5886.png,Lung_Opacity/images/Lung_Opacity-5886.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5886.png,Lung_Opacity/masks/Lung_Opacity-5886.png,True,False,299,299,L,119.37,c06bcecccba354f16e8733e8bfdf365f
Lung_Opacity-5887.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5887.png,Lung_Opacity/images/Lung_Opacity-5887.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5887.png,Lung_Opacity/masks/Lung_Opacity-5887.png,True,False,299,299,L,179.95,3ce5182d7d689c9e0a54e7ef3e0105b1
Lung_Opacity-5888.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5888.png,Lung_Opacity/images/Lung_Opacity-5888.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5888.png,Lung_Opacity/masks/Lung_Opacity-5888.png,True,False,299,299,L,133.38,3e100c01a3df02d6db71e2859d8ceb49
Lung_Opacity-5889.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5889.png,Lung_Opacity/images/Lung_Opacity-5889.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5889.png,Lung_Opacity/masks/Lung_Opacity-5889.png,True,False,299,299,L,170.4,bd5ebabd86b934b1c0efa60c28745ebe
Lung_Opacity-589.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-589.png,Lung_Opacity/images/Lung_Opacity-589.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-589.png,Lung_Opacity/masks/Lung_Opacity-589.png,True,False,299,299,L,87.15,153cdb83334365a1a6fe0ea2aca7c644
Lung_Opacity-5890.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5890.png,Lung_Opacity/images/Lung_Opacity-5890.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5890.png,Lung_Opacity/masks/Lung_Opacity-5890.png,True,False,299,299,L,150.45,a3812af7a148d7b28553df63dc218d71
Lung_Opacity-5891.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5891.png,Lung_Opacity/images/Lung_Opacity-5891.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5891.png,Lung_Opacity/masks/Lung_Opacity-5891.png,True,False,299,299,L,112.81,6ff7a311acb7f8779979bc52bb37cb9e
Lung_Opacity-5892.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5892.png,Lung_Opacity/images/Lung_Opacity-5892.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5892.png,Lung_Opacity/masks/Lung_Opacity-5892.png,True,False,299,299,L,140.42,c829485c6412fe25458c93b37b38ce59
Lung_Opacity-5893.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5893.png,Lung_Opacity/images/Lung_Opacity-5893.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5893.png,Lung_Opacity/masks/Lung_Opacity-5893.png,True,False,299,299,L,155.95,08d2935d355e66fb2d451c5a003c1eaa
Lung_Opacity-5894.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5894.png,Lung_Opacity/images/Lung_Opacity-5894.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5894.png,Lung_Opacity/masks/Lung_Opacity-5894.png,True,False,299,299,L,117.21,f3acb3d549fff8684cc604befdb6ed63
Lung_Opacity-5895.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5895.png,Lung_Opacity/images/Lung_Opacity-5895.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5895.png,Lung_Opacity/masks/Lung_Opacity-5895.png,True,False,299,299,L,117.61,9aa876622ee63a858eb37256123a12fb
Lung_Opacity-5896.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5896.png,Lung_Opacity/images/Lung_Opacity-5896.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5896.png,Lung_Opacity/masks/Lung_Opacity-5896.png,True,False,299,299,L,107.52,3115bfe9bfa331ccccaa4861f040004a
Lung_Opacity-5897.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5897.png,Lung_Opacity/images/Lung_Opacity-5897.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5897.png,Lung_Opacity/masks/Lung_Opacity-5897.png,True,False,299,299,L,157.13,fa80ac0c91758e4ce40e115100ac849a
Lung_Opacity-5898.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5898.png,Lung_Opacity/images/Lung_Opacity-5898.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5898.png,Lung_Opacity/masks/Lung_Opacity-5898.png,True,False,299,299,L,106.77,8689005fa4cf9c4984114705e2533807
Lung_Opacity-5899.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5899.png,Lung_Opacity/images/Lung_Opacity-5899.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5899.png,Lung_Opacity/masks/Lung_Opacity-5899.png,True,False,299,299,L,129.02,363c0116bb0e0487b2c069d695b3e84d
Lung_Opacity-59.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-59.png,Lung_Opacity/images/Lung_Opacity-59.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-59.png,Lung_Opacity/masks/Lung_Opacity-59.png,True,False,299,299,L,118.18,dbee9795341a8aa59623e209f7af8401
Lung_Opacity-590.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-590.png,Lung_Opacity/images/Lung_Opacity-590.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-590.png,Lung_Opacity/masks/Lung_Opacity-590.png,True,False,299,299,L,114.5,dbce9e09e86a291ca2c96b5c60f7d871
Lung_Opacity-5900.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5900.png,Lung_Opacity/images/Lung_Opacity-5900.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5900.png,Lung_Opacity/masks/Lung_Opacity-5900.png,True,False,299,299,L,145.14,5fab34a91d8eab7be22eeab8516ad4dc
Lung_Opacity-5901.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5901.png,Lung_Opacity/images/Lung_Opacity-5901.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5901.png,Lung_Opacity/masks/Lung_Opacity-5901.png,True,False,299,299,L,165.03,d1f16cafdb8d2245f8baae5216d8d99c
Lung_Opacity-5902.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5902.png,Lung_Opacity/images/Lung_Opacity-5902.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5902.png,Lung_Opacity/masks/Lung_Opacity-5902.png,True,False,299,299,L,170.41,2e6a9123e642cd3877817174e511b6ea
Lung_Opacity-5903.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5903.png,Lung_Opacity/images/Lung_Opacity-5903.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5903.png,Lung_Opacity/masks/Lung_Opacity-5903.png,True,False,299,299,L,125.0,b1711dcfe5f910754e7a01edca6cafa6
Lung_Opacity-5904.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5904.png,Lung_Opacity/images/Lung_Opacity-5904.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5904.png,Lung_Opacity/masks/Lung_Opacity-5904.png,True,False,299,299,L,116.13,3dfc470e88664fbe66bb31e65c32b24c
Lung_Opacity-5905.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5905.png,Lung_Opacity/images/Lung_Opacity-5905.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5905.png,Lung_Opacity/masks/Lung_Opacity-5905.png,True,False,299,299,L,120.95,76b240300f28b88c45986f3dc7a3f63e
Lung_Opacity-5906.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5906.png,Lung_Opacity/images/Lung_Opacity-5906.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5906.png,Lung_Opacity/masks/Lung_Opacity-5906.png,True,False,299,299,L,101.97,ca7a8a3f76598e3aaaa4ee5107ae0bac
Lung_Opacity-5907.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5907.png,Lung_Opacity/images/Lung_Opacity-5907.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5907.png,Lung_Opacity/masks/Lung_Opacity-5907.png,True,False,299,299,L,100.85,1b783342f91eb4e7ede9c77000a00a56
Lung_Opacity-5908.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5908.png,Lung_Opacity/images/Lung_Opacity-5908.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5908.png,Lung_Opacity/masks/Lung_Opacity-5908.png,True,False,299,299,L,146.54,50fc3601d17ffddb20354ad397eb8f74
Lung_Opacity-5909.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5909.png,Lung_Opacity/images/Lung_Opacity-5909.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5909.png,Lung_Opacity/masks/Lung_Opacity-5909.png,True,False,299,299,L,123.43,2a1f047f613b8df956322a5afebfcf48
Lung_Opacity-591.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-591.png,Lung_Opacity/images/Lung_Opacity-591.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-591.png,Lung_Opacity/masks/Lung_Opacity-591.png,True,False,299,299,L,175.09,ee769eb55a5d07359ed09b4dbc9d3402
Lung_Opacity-5910.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5910.png,Lung_Opacity/images/Lung_Opacity-5910.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5910.png,Lung_Opacity/masks/Lung_Opacity-5910.png,True,False,299,299,L,158.36,d999396d1830e3d862cd8d0418389f79
Lung_Opacity-5911.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5911.png,Lung_Opacity/images/Lung_Opacity-5911.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5911.png,Lung_Opacity/masks/Lung_Opacity-5911.png,True,False,299,299,L,145.27,7f274a0f1e80290c09f21655970df071
Lung_Opacity-5912.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5912.png,Lung_Opacity/images/Lung_Opacity-5912.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5912.png,Lung_Opacity/masks/Lung_Opacity-5912.png,True,False,299,299,L,110.31,de30bcdbf8d04a7b953c980c6e385984
Lung_Opacity-5913.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5913.png,Lung_Opacity/images/Lung_Opacity-5913.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5913.png,Lung_Opacity/masks/Lung_Opacity-5913.png,True,False,299,299,L,117.78,8a388779f0f6b72283799ad99322e8df
Lung_Opacity-5914.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5914.png,Lung_Opacity/images/Lung_Opacity-5914.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5914.png,Lung_Opacity/masks/Lung_Opacity-5914.png,True,False,299,299,L,127.61,b56bd9f821440c72f9e4cca2ec2c187a
Lung_Opacity-5915.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5915.png,Lung_Opacity/images/Lung_Opacity-5915.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5915.png,Lung_Opacity/masks/Lung_Opacity-5915.png,True,False,299,299,L,137.31,f4d67e9dbfc645d6e83071efe61234e3
Lung_Opacity-5916.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5916.png,Lung_Opacity/images/Lung_Opacity-5916.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5916.png,Lung_Opacity/masks/Lung_Opacity-5916.png,True,False,299,299,L,127.9,328aa1b6430de1aa8b035d87eac20e91
Lung_Opacity-5917.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5917.png,Lung_Opacity/images/Lung_Opacity-5917.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5917.png,Lung_Opacity/masks/Lung_Opacity-5917.png,True,False,299,299,L,142.7,d0920310444f9c44c96e46faae2d79bc
Lung_Opacity-5918.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5918.png,Lung_Opacity/images/Lung_Opacity-5918.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5918.png,Lung_Opacity/masks/Lung_Opacity-5918.png,True,False,299,299,L,118.19,0703422bfbb706cb7109e6483b11677e
Lung_Opacity-5919.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5919.png,Lung_Opacity/images/Lung_Opacity-5919.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5919.png,Lung_Opacity/masks/Lung_Opacity-5919.png,True,False,299,299,L,140.94,ad6907f0b15d1b54880914eb6f6dea49
Lung_Opacity-592.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-592.png,Lung_Opacity/images/Lung_Opacity-592.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-592.png,Lung_Opacity/masks/Lung_Opacity-592.png,True,False,299,299,L,126.96,f068cf79a55f6655da6620cefe6b14a8
Lung_Opacity-5920.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5920.png,Lung_Opacity/images/Lung_Opacity-5920.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5920.png,Lung_Opacity/masks/Lung_Opacity-5920.png,True,False,299,299,L,100.62,3f67fdbecacc01095aff94d69d5eb519
Lung_Opacity-5921.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5921.png,Lung_Opacity/images/Lung_Opacity-5921.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5921.png,Lung_Opacity/masks/Lung_Opacity-5921.png,True,False,299,299,L,107.83,82c282e590500588a439dac81247f302
Lung_Opacity-5922.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5922.png,Lung_Opacity/images/Lung_Opacity-5922.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5922.png,Lung_Opacity/masks/Lung_Opacity-5922.png,True,False,299,299,L,110.71,d599d94162fd0d86aa4161d77189d711
Lung_Opacity-5923.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5923.png,Lung_Opacity/images/Lung_Opacity-5923.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5923.png,Lung_Opacity/masks/Lung_Opacity-5923.png,True,False,299,299,L,168.68,629e3966c5e963b25129d09f924b16f4
Lung_Opacity-5924.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5924.png,Lung_Opacity/images/Lung_Opacity-5924.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5924.png,Lung_Opacity/masks/Lung_Opacity-5924.png,True,False,299,299,L,109.86,ad68b7a88048ae51164bead023c065be
Lung_Opacity-5925.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5925.png,Lung_Opacity/images/Lung_Opacity-5925.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5925.png,Lung_Opacity/masks/Lung_Opacity-5925.png,True,False,299,299,L,107.34,e8185a2eee0f62d596917f1275d585e4
Lung_Opacity-5926.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5926.png,Lung_Opacity/images/Lung_Opacity-5926.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5926.png,Lung_Opacity/masks/Lung_Opacity-5926.png,True,False,299,299,L,110.1,2b2dcaa372deef7a40eb2ddac470bf5a
Lung_Opacity-5927.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5927.png,Lung_Opacity/images/Lung_Opacity-5927.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5927.png,Lung_Opacity/masks/Lung_Opacity-5927.png,True,False,299,299,L,126.09,3a95041abcf85ca87e133c89a325feaa
Lung_Opacity-5928.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5928.png,Lung_Opacity/images/Lung_Opacity-5928.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5928.png,Lung_Opacity/masks/Lung_Opacity-5928.png,True,False,299,299,L,125.36,5574604ba95f91ef0f47542bf4ca286a
Lung_Opacity-5929.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5929.png,Lung_Opacity/images/Lung_Opacity-5929.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5929.png,Lung_Opacity/masks/Lung_Opacity-5929.png,True,False,299,299,L,118.13,b9ddb2061e68a2a6e2f7487cf8529a82
Lung_Opacity-593.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-593.png,Lung_Opacity/images/Lung_Opacity-593.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-593.png,Lung_Opacity/masks/Lung_Opacity-593.png,True,False,299,299,L,132.84,7994400b36131b8726c794cd59babc85
Lung_Opacity-5930.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5930.png,Lung_Opacity/images/Lung_Opacity-5930.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5930.png,Lung_Opacity/masks/Lung_Opacity-5930.png,True,False,299,299,L,141.51,1d39f03955c2bb5c5fabb26ff2a648b4
Lung_Opacity-5931.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5931.png,Lung_Opacity/images/Lung_Opacity-5931.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5931.png,Lung_Opacity/masks/Lung_Opacity-5931.png,True,False,299,299,L,112.61,0f880d9349af82010871c2279abdb709
Lung_Opacity-5932.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5932.png,Lung_Opacity/images/Lung_Opacity-5932.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5932.png,Lung_Opacity/masks/Lung_Opacity-5932.png,True,False,299,299,L,170.15,a41e717644a87e78b8d9050497429189
Lung_Opacity-5933.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5933.png,Lung_Opacity/images/Lung_Opacity-5933.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5933.png,Lung_Opacity/masks/Lung_Opacity-5933.png,True,False,299,299,L,143.43,a5642779cd45b19f0cf3854928d4de96
Lung_Opacity-5934.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5934.png,Lung_Opacity/images/Lung_Opacity-5934.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5934.png,Lung_Opacity/masks/Lung_Opacity-5934.png,True,False,299,299,L,116.4,370f14822f424a7d36f3ec4e352f02e4
Lung_Opacity-5935.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5935.png,Lung_Opacity/images/Lung_Opacity-5935.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5935.png,Lung_Opacity/masks/Lung_Opacity-5935.png,True,False,299,299,L,165.88,8142535046d53fd53dacffc900d20916
Lung_Opacity-5936.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5936.png,Lung_Opacity/images/Lung_Opacity-5936.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5936.png,Lung_Opacity/masks/Lung_Opacity-5936.png,True,False,299,299,L,115.09,452030d3a0c874846eba5328f18b806e
Lung_Opacity-5937.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5937.png,Lung_Opacity/images/Lung_Opacity-5937.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5937.png,Lung_Opacity/masks/Lung_Opacity-5937.png,True,False,299,299,L,123.6,c9ba3bbbfe9b0c75d88f5a0e23f978c0
Lung_Opacity-5938.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5938.png,Lung_Opacity/images/Lung_Opacity-5938.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5938.png,Lung_Opacity/masks/Lung_Opacity-5938.png,True,False,299,299,L,157.24,e4ab18f6b0123e9a6c59a1127ff427e5
Lung_Opacity-5939.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5939.png,Lung_Opacity/images/Lung_Opacity-5939.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5939.png,Lung_Opacity/masks/Lung_Opacity-5939.png,True,False,299,299,L,115.48,2671ba4e68485c5d86da79c55b582555
Lung_Opacity-594.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-594.png,Lung_Opacity/images/Lung_Opacity-594.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-594.png,Lung_Opacity/masks/Lung_Opacity-594.png,True,False,299,299,L,143.76,ca70ea34b83e454adf2166d05f00d436
Lung_Opacity-5940.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5940.png,Lung_Opacity/images/Lung_Opacity-5940.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5940.png,Lung_Opacity/masks/Lung_Opacity-5940.png,True,False,299,299,L,125.7,7d6daaba0fdf688b45d07ce2b159a26d
Lung_Opacity-5941.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5941.png,Lung_Opacity/images/Lung_Opacity-5941.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5941.png,Lung_Opacity/masks/Lung_Opacity-5941.png,True,False,299,299,L,159.92,57f06036554fa48fe632149e0dda9df1
Lung_Opacity-5942.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5942.png,Lung_Opacity/images/Lung_Opacity-5942.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5942.png,Lung_Opacity/masks/Lung_Opacity-5942.png,True,False,299,299,L,164.59,8066f0d0469c28ca29aacf96928a9eaa
Lung_Opacity-5943.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5943.png,Lung_Opacity/images/Lung_Opacity-5943.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5943.png,Lung_Opacity/masks/Lung_Opacity-5943.png,True,False,299,299,L,136.05,cfc95cafecac1d9466f20c93350c954c
Lung_Opacity-5944.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5944.png,Lung_Opacity/images/Lung_Opacity-5944.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5944.png,Lung_Opacity/masks/Lung_Opacity-5944.png,True,False,299,299,L,118.24,d9194017d0eacbf2bb26daf954ccf278
Lung_Opacity-5945.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5945.png,Lung_Opacity/images/Lung_Opacity-5945.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5945.png,Lung_Opacity/masks/Lung_Opacity-5945.png,True,False,299,299,L,149.84,2e8756bb619834e96d5a9bfb2e77a26f
Lung_Opacity-5946.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5946.png,Lung_Opacity/images/Lung_Opacity-5946.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5946.png,Lung_Opacity/masks/Lung_Opacity-5946.png,True,False,299,299,L,119.9,f06db15b67cb5ec6422e8db0162baa76
Lung_Opacity-5947.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5947.png,Lung_Opacity/images/Lung_Opacity-5947.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5947.png,Lung_Opacity/masks/Lung_Opacity-5947.png,True,False,299,299,L,163.27,f1f28645503e1e2870dc56971f6f900d
Lung_Opacity-5948.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5948.png,Lung_Opacity/images/Lung_Opacity-5948.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5948.png,Lung_Opacity/masks/Lung_Opacity-5948.png,True,False,299,299,L,109.24,a3198fb9d42dea98ebb4064d6fe93c70
Lung_Opacity-5949.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5949.png,Lung_Opacity/images/Lung_Opacity-5949.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5949.png,Lung_Opacity/masks/Lung_Opacity-5949.png,True,False,299,299,L,121.68,4f930a1dc12400d98acb25bc7f04ff75
Lung_Opacity-595.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-595.png,Lung_Opacity/images/Lung_Opacity-595.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-595.png,Lung_Opacity/masks/Lung_Opacity-595.png,True,False,299,299,L,164.44,d6e505adffd65ec975e43c75423ca7b8
Lung_Opacity-5950.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5950.png,Lung_Opacity/images/Lung_Opacity-5950.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5950.png,Lung_Opacity/masks/Lung_Opacity-5950.png,True,False,299,299,L,145.54,bf28c4c2b55924db7e691c4ce1737b0a
Lung_Opacity-5951.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5951.png,Lung_Opacity/images/Lung_Opacity-5951.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5951.png,Lung_Opacity/masks/Lung_Opacity-5951.png,True,False,299,299,L,135.92,923f85199eecd67a5730782693220dd9
Lung_Opacity-5952.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5952.png,Lung_Opacity/images/Lung_Opacity-5952.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5952.png,Lung_Opacity/masks/Lung_Opacity-5952.png,True,False,299,299,L,104.37,a32e6f4a315e96a7c07a0d975515a62b
Lung_Opacity-5953.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5953.png,Lung_Opacity/images/Lung_Opacity-5953.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5953.png,Lung_Opacity/masks/Lung_Opacity-5953.png,True,False,299,299,L,181.77,f7a81e71d364af714ccd3068535bf89d
Lung_Opacity-5954.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5954.png,Lung_Opacity/images/Lung_Opacity-5954.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5954.png,Lung_Opacity/masks/Lung_Opacity-5954.png,True,False,299,299,L,117.3,aa03007d832012b19a6c540ccc64e178
Lung_Opacity-5955.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5955.png,Lung_Opacity/images/Lung_Opacity-5955.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5955.png,Lung_Opacity/masks/Lung_Opacity-5955.png,True,False,299,299,L,120.4,a4bda4413633c01e5af80e866832a083
Lung_Opacity-5956.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5956.png,Lung_Opacity/images/Lung_Opacity-5956.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5956.png,Lung_Opacity/masks/Lung_Opacity-5956.png,True,False,299,299,L,125.03,354ddfda9ed09cd362db6fb97059d26f
Lung_Opacity-5957.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5957.png,Lung_Opacity/images/Lung_Opacity-5957.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5957.png,Lung_Opacity/masks/Lung_Opacity-5957.png,True,False,299,299,L,108.68,f7021e18d137c2bc45c4cbc99c87c2b3
Lung_Opacity-5958.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5958.png,Lung_Opacity/images/Lung_Opacity-5958.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5958.png,Lung_Opacity/masks/Lung_Opacity-5958.png,True,False,299,299,L,161.26,e3e78d4dd9e6dd5476d90b518226c187
Lung_Opacity-5959.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5959.png,Lung_Opacity/images/Lung_Opacity-5959.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5959.png,Lung_Opacity/masks/Lung_Opacity-5959.png,True,False,299,299,L,105.95,7b9e762feafedae880ad6a6c6745737c
Lung_Opacity-596.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-596.png,Lung_Opacity/images/Lung_Opacity-596.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-596.png,Lung_Opacity/masks/Lung_Opacity-596.png,True,False,299,299,L,148.55,00001a8511fe190756f9e6744de2de11
Lung_Opacity-5960.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5960.png,Lung_Opacity/images/Lung_Opacity-5960.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5960.png,Lung_Opacity/masks/Lung_Opacity-5960.png,True,False,299,299,L,112.93,2142f8d61e160839a0efbfb46c36a972
Lung_Opacity-5961.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5961.png,Lung_Opacity/images/Lung_Opacity-5961.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5961.png,Lung_Opacity/masks/Lung_Opacity-5961.png,True,False,299,299,L,118.67,7037c721ceaf69280d9db821586cb5ed
Lung_Opacity-5962.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5962.png,Lung_Opacity/images/Lung_Opacity-5962.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5962.png,Lung_Opacity/masks/Lung_Opacity-5962.png,True,False,299,299,L,137.06,96216a557cbda0008b9128c8c7dbdbd4
Lung_Opacity-5963.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5963.png,Lung_Opacity/images/Lung_Opacity-5963.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5963.png,Lung_Opacity/masks/Lung_Opacity-5963.png,True,False,299,299,L,114.58,32470f8decaffd5519341392306b9827
Lung_Opacity-5964.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5964.png,Lung_Opacity/images/Lung_Opacity-5964.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5964.png,Lung_Opacity/masks/Lung_Opacity-5964.png,True,False,299,299,L,137.33,8693dc5cd2550049eec9e390a893f988
Lung_Opacity-5965.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5965.png,Lung_Opacity/images/Lung_Opacity-5965.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5965.png,Lung_Opacity/masks/Lung_Opacity-5965.png,True,False,299,299,L,123.7,13ba741faed0fc0fafbb996d276e2694
Lung_Opacity-5966.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5966.png,Lung_Opacity/images/Lung_Opacity-5966.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5966.png,Lung_Opacity/masks/Lung_Opacity-5966.png,True,False,299,299,L,109.37,1617a1e2e00962fae66efd7686256fe8
Lung_Opacity-5967.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5967.png,Lung_Opacity/images/Lung_Opacity-5967.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5967.png,Lung_Opacity/masks/Lung_Opacity-5967.png,True,False,299,299,L,163.08,ebc66bd2ebc5fd0e9878ddb579bf967f
Lung_Opacity-5968.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5968.png,Lung_Opacity/images/Lung_Opacity-5968.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5968.png,Lung_Opacity/masks/Lung_Opacity-5968.png,True,False,299,299,L,168.83,7f6250dd43be1ee940dbc7aa9369446a
Lung_Opacity-5969.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5969.png,Lung_Opacity/images/Lung_Opacity-5969.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5969.png,Lung_Opacity/masks/Lung_Opacity-5969.png,True,False,299,299,L,137.61,d21aa0de3cd18079008aa5b2204768bb
Lung_Opacity-597.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-597.png,Lung_Opacity/images/Lung_Opacity-597.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-597.png,Lung_Opacity/masks/Lung_Opacity-597.png,True,False,299,299,L,168.83,f3faf876e8a16e25e7cecc5f145bb6d8
Lung_Opacity-5970.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5970.png,Lung_Opacity/images/Lung_Opacity-5970.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5970.png,Lung_Opacity/masks/Lung_Opacity-5970.png,True,False,299,299,L,122.28,5756c81e20501413f7c7e099bbf45d7c
Lung_Opacity-5971.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5971.png,Lung_Opacity/images/Lung_Opacity-5971.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5971.png,Lung_Opacity/masks/Lung_Opacity-5971.png,True,False,299,299,L,151.4,bf0e8ab93c330570e530b9ffaf727e3b
Lung_Opacity-5972.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5972.png,Lung_Opacity/images/Lung_Opacity-5972.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5972.png,Lung_Opacity/masks/Lung_Opacity-5972.png,True,False,299,299,L,118.51,0162e6eaf5acb29f545cb7054b33da2b
Lung_Opacity-5973.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5973.png,Lung_Opacity/images/Lung_Opacity-5973.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5973.png,Lung_Opacity/masks/Lung_Opacity-5973.png,True,False,299,299,L,88.24,66b3a3260fbf96dbe21b780db0f6ac4e
Lung_Opacity-5974.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5974.png,Lung_Opacity/images/Lung_Opacity-5974.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5974.png,Lung_Opacity/masks/Lung_Opacity-5974.png,True,False,299,299,L,120.87,97de29da7fcc2330695926288b7e8eac
Lung_Opacity-5975.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5975.png,Lung_Opacity/images/Lung_Opacity-5975.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5975.png,Lung_Opacity/masks/Lung_Opacity-5975.png,True,False,299,299,L,165.96,8a488f92f35900dde3f98faf867481e7
Lung_Opacity-5976.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5976.png,Lung_Opacity/images/Lung_Opacity-5976.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5976.png,Lung_Opacity/masks/Lung_Opacity-5976.png,True,False,299,299,L,107.78,af0a9cacf037693d5da62b33d67334fb
Lung_Opacity-5977.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5977.png,Lung_Opacity/images/Lung_Opacity-5977.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5977.png,Lung_Opacity/masks/Lung_Opacity-5977.png,True,False,299,299,L,160.8,9436b5741e7482777af6c5c18a6f4e39
Lung_Opacity-5978.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5978.png,Lung_Opacity/images/Lung_Opacity-5978.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5978.png,Lung_Opacity/masks/Lung_Opacity-5978.png,True,False,299,299,L,138.68,8b514c6e3dcdc2239adfa410f47cb02e
Lung_Opacity-5979.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5979.png,Lung_Opacity/images/Lung_Opacity-5979.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5979.png,Lung_Opacity/masks/Lung_Opacity-5979.png,True,False,299,299,L,131.49,80268c0aa48ea75bea91a3f984cad1f8
Lung_Opacity-598.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-598.png,Lung_Opacity/images/Lung_Opacity-598.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-598.png,Lung_Opacity/masks/Lung_Opacity-598.png,True,False,299,299,L,134.72,038dda2f9765d14de7ba19065991654e
Lung_Opacity-5980.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5980.png,Lung_Opacity/images/Lung_Opacity-5980.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5980.png,Lung_Opacity/masks/Lung_Opacity-5980.png,True,False,299,299,L,166.49,b2e2f8da6730e6586c049d533b818849
Lung_Opacity-5981.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5981.png,Lung_Opacity/images/Lung_Opacity-5981.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5981.png,Lung_Opacity/masks/Lung_Opacity-5981.png,True,False,299,299,L,149.59,dbe6f17e48bf885885115d843866ba94
Lung_Opacity-5982.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5982.png,Lung_Opacity/images/Lung_Opacity-5982.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5982.png,Lung_Opacity/masks/Lung_Opacity-5982.png,True,False,299,299,L,132.11,090352493b6f571f1aad5527168674e0
Lung_Opacity-5983.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5983.png,Lung_Opacity/images/Lung_Opacity-5983.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5983.png,Lung_Opacity/masks/Lung_Opacity-5983.png,True,False,299,299,L,169.04,c2199860a7eb870ba1fbc9db9bb9f076
Lung_Opacity-5984.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5984.png,Lung_Opacity/images/Lung_Opacity-5984.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5984.png,Lung_Opacity/masks/Lung_Opacity-5984.png,True,False,299,299,L,121.73,bc75264702c41b951e504244252d3549
Lung_Opacity-5985.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5985.png,Lung_Opacity/images/Lung_Opacity-5985.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5985.png,Lung_Opacity/masks/Lung_Opacity-5985.png,True,False,299,299,L,119.44,d3351dd194d1103f435d606e06db3910
Lung_Opacity-5986.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5986.png,Lung_Opacity/images/Lung_Opacity-5986.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5986.png,Lung_Opacity/masks/Lung_Opacity-5986.png,True,False,299,299,L,117.0,c8b860f3ce390bbbf4ecfffd29b546f2
Lung_Opacity-5987.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5987.png,Lung_Opacity/images/Lung_Opacity-5987.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5987.png,Lung_Opacity/masks/Lung_Opacity-5987.png,True,False,299,299,L,162.44,842cd0093508271f6a9cb0203c25179c
Lung_Opacity-5988.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5988.png,Lung_Opacity/images/Lung_Opacity-5988.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5988.png,Lung_Opacity/masks/Lung_Opacity-5988.png,True,False,299,299,L,129.16,fae63221af481d68524674e2745a3e75
Lung_Opacity-5989.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5989.png,Lung_Opacity/images/Lung_Opacity-5989.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5989.png,Lung_Opacity/masks/Lung_Opacity-5989.png,True,False,299,299,L,122.68,2d0ad9c700e03562b699e9544e7719b4
Lung_Opacity-599.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-599.png,Lung_Opacity/images/Lung_Opacity-599.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-599.png,Lung_Opacity/masks/Lung_Opacity-599.png,True,False,299,299,L,139.45,1dbe04d92ef407993d7994408b608fa1
Lung_Opacity-5990.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5990.png,Lung_Opacity/images/Lung_Opacity-5990.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5990.png,Lung_Opacity/masks/Lung_Opacity-5990.png,True,False,299,299,L,119.17,8382c81811ef117dcb5a6ede217397d3
Lung_Opacity-5991.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5991.png,Lung_Opacity/images/Lung_Opacity-5991.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5991.png,Lung_Opacity/masks/Lung_Opacity-5991.png,True,False,299,299,L,146.12,2ff834b35a2c84209f0b5d4ebc1fce32
Lung_Opacity-5992.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5992.png,Lung_Opacity/images/Lung_Opacity-5992.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5992.png,Lung_Opacity/masks/Lung_Opacity-5992.png,True,False,299,299,L,108.45,59c4d852cfa397fbcd6e90ad7478b9d7
Lung_Opacity-5993.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5993.png,Lung_Opacity/images/Lung_Opacity-5993.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5993.png,Lung_Opacity/masks/Lung_Opacity-5993.png,True,False,299,299,L,102.45,3ccc9160ba47f3233d7290765874d840
Lung_Opacity-5994.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5994.png,Lung_Opacity/images/Lung_Opacity-5994.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5994.png,Lung_Opacity/masks/Lung_Opacity-5994.png,True,False,299,299,L,136.86,25fcc1a8fefcb103f18a54bfea46bf71
Lung_Opacity-5995.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5995.png,Lung_Opacity/images/Lung_Opacity-5995.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5995.png,Lung_Opacity/masks/Lung_Opacity-5995.png,True,False,299,299,L,164.81,5fcdc79564a10aa5f8ceebebb3704223
Lung_Opacity-5996.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5996.png,Lung_Opacity/images/Lung_Opacity-5996.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5996.png,Lung_Opacity/masks/Lung_Opacity-5996.png,True,False,299,299,L,120.9,52fe86c169a438e590683bf27fcb98e8
Lung_Opacity-5997.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5997.png,Lung_Opacity/images/Lung_Opacity-5997.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5997.png,Lung_Opacity/masks/Lung_Opacity-5997.png,True,False,299,299,L,109.16,7ed7010fbac3c5c5ffed3f250ffdcd62
Lung_Opacity-5998.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5998.png,Lung_Opacity/images/Lung_Opacity-5998.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5998.png,Lung_Opacity/masks/Lung_Opacity-5998.png,True,False,299,299,L,128.45,ef85729c02ea8bb38263e8436e51b1ec
Lung_Opacity-5999.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-5999.png,Lung_Opacity/images/Lung_Opacity-5999.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-5999.png,Lung_Opacity/masks/Lung_Opacity-5999.png,True,False,299,299,L,115.15,463221548694f48bc9d80bfca29c72dc
Lung_Opacity-6.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-6.png,Lung_Opacity/images/Lung_Opacity-6.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-6.png,Lung_Opacity/masks/Lung_Opacity-6.png,True,False,299,299,L,119.88,5fc358d227267cf09970fe2be9957901
Lung_Opacity-60.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-60.png,Lung_Opacity/images/Lung_Opacity-60.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-60.png,Lung_Opacity/masks/Lung_Opacity-60.png,True,False,299,299,L,161.48,13f8952478f42ec7719c0ac1d6ffc547
Lung_Opacity-600.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-600.png,Lung_Opacity/images/Lung_Opacity-600.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-600.png,Lung_Opacity/masks/Lung_Opacity-600.png,True,False,299,299,L,113.23,313de802183d45db147460d5a9f54825
Lung_Opacity-6000.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-6000.png,Lung_Opacity/images/Lung_Opacity-6000.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-6000.png,Lung_Opacity/masks/Lung_Opacity-6000.png,True,False,299,299,L,110.57,af3b3bdfcabbfab68bdf4308042e21f8
Lung_Opacity-6001.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-6001.png,Lung_Opacity/images/Lung_Opacity-6001.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-6001.png,Lung_Opacity/masks/Lung_Opacity-6001.png,True,False,299,299,L,131.21,1b8245ec63a2062347ef3113d3cd6d78
Lung_Opacity-6002.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-6002.png,Lung_Opacity/images/Lung_Opacity-6002.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-6002.png,Lung_Opacity/masks/Lung_Opacity-6002.png,True,False,299,299,L,141.26,7c9d03449cbd0b0f2a156b5c9e5ffb67
Lung_Opacity-6003.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-6003.png,Lung_Opacity/images/Lung_Opacity-6003.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-6003.png,Lung_Opacity/masks/Lung_Opacity-6003.png,True,False,299,299,L,132.65,5b6c2b351121f6659c30a09e79e688a6
Lung_Opacity-6004.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-6004.png,Lung_Opacity/images/Lung_Opacity-6004.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-6004.png,Lung_Opacity/masks/Lung_Opacity-6004.png,True,False,299,299,L,111.64,92b4b7eec56c2a0381c72c863c7d1d5c
Lung_Opacity-6005.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-6005.png,Lung_Opacity/images/Lung_Opacity-6005.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-6005.png,Lung_Opacity/masks/Lung_Opacity-6005.png,True,False,299,299,L,41.5,11fcee9149f54e3355bfd8859f131b27
Lung_Opacity-6006.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-6006.png,Lung_Opacity/images/Lung_Opacity-6006.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-6006.png,Lung_Opacity/masks/Lung_Opacity-6006.png,True,False,299,299,L,112.56,4c9ce0ad0f015f44a8afed6c30f2a5f7
Lung_Opacity-6007.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-6007.png,Lung_Opacity/images/Lung_Opacity-6007.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-6007.png,Lung_Opacity/masks/Lung_Opacity-6007.png,True,False,299,299,L,134.73,a101dcf7bb27e686a7b85150c4d7f591
Lung_Opacity-6008.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-6008.png,Lung_Opacity/images/Lung_Opacity-6008.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-6008.png,Lung_Opacity/masks/Lung_Opacity-6008.png,True,False,299,299,L,110.66,b3aa42162675b900fd1afd1c62993b1d
Lung_Opacity-6009.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-6009.png,Lung_Opacity/images/Lung_Opacity-6009.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-6009.png,Lung_Opacity/masks/Lung_Opacity-6009.png,True,False,299,299,L,121.38,057058648a94e2738152b9d3d60de812
Lung_Opacity-601.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-601.png,Lung_Opacity/images/Lung_Opacity-601.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-601.png,Lung_Opacity/masks/Lung_Opacity-601.png,True,False,299,299,L,161.48,f04745b614b9b6bc7da163e3dd7d9e2e
Lung_Opacity-6010.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-6010.png,Lung_Opacity/images/Lung_Opacity-6010.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-6010.png,Lung_Opacity/masks/Lung_Opacity-6010.png,True,False,299,299,L,127.75,0c822069f89410f9e38bb9c0b61dffa7
Lung_Opacity-6011.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-6011.png,Lung_Opacity/images/Lung_Opacity-6011.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-6011.png,Lung_Opacity/masks/Lung_Opacity-6011.png,True,False,299,299,L,120.21,800ad56fa50b7d631cb8a313162eb246
Lung_Opacity-6012.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-6012.png,Lung_Opacity/images/Lung_Opacity-6012.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-6012.png,Lung_Opacity/masks/Lung_Opacity-6012.png,True,False,299,299,L,141.39,3c2a1cbb681cef159e086c6051b21437
Lung_Opacity-602.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-602.png,Lung_Opacity/images/Lung_Opacity-602.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-602.png,Lung_Opacity/masks/Lung_Opacity-602.png,True,False,299,299,L,116.72,5476c10f491df3c632d5afa2caacd5fb
Lung_Opacity-603.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-603.png,Lung_Opacity/images/Lung_Opacity-603.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-603.png,Lung_Opacity/masks/Lung_Opacity-603.png,True,False,299,299,L,116.75,a59ecc2470f76bb498d43045955aacb1
Lung_Opacity-604.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-604.png,Lung_Opacity/images/Lung_Opacity-604.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-604.png,Lung_Opacity/masks/Lung_Opacity-604.png,True,False,299,299,L,143.01,0c0524f12822fda0eb3727075c81ec8a
Lung_Opacity-605.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-605.png,Lung_Opacity/images/Lung_Opacity-605.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-605.png,Lung_Opacity/masks/Lung_Opacity-605.png,True,False,299,299,L,98.68,e9244c64349314a48db98cb277e48e5c
Lung_Opacity-606.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-606.png,Lung_Opacity/images/Lung_Opacity-606.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-606.png,Lung_Opacity/masks/Lung_Opacity-606.png,True,False,299,299,L,116.88,738f8ad59ece3240c777f53b40b6f747
Lung_Opacity-607.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-607.png,Lung_Opacity/images/Lung_Opacity-607.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-607.png,Lung_Opacity/masks/Lung_Opacity-607.png,True,False,299,299,L,108.34,8f3989a550dbda5a2a163fcefc0a9801
Lung_Opacity-608.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-608.png,Lung_Opacity/images/Lung_Opacity-608.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-608.png,Lung_Opacity/masks/Lung_Opacity-608.png,True,False,299,299,L,172.16,22be4558c80612e980367a80050bb65e
Lung_Opacity-609.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-609.png,Lung_Opacity/images/Lung_Opacity-609.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-609.png,Lung_Opacity/masks/Lung_Opacity-609.png,True,False,299,299,L,122.02,2556b0eaa65b09d29ba81b15c0fe1fba
Lung_Opacity-61.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-61.png,Lung_Opacity/images/Lung_Opacity-61.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-61.png,Lung_Opacity/masks/Lung_Opacity-61.png,True,False,299,299,L,112.09,71667d4e410220021c95240bc5373685
Lung_Opacity-610.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-610.png,Lung_Opacity/images/Lung_Opacity-610.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-610.png,Lung_Opacity/masks/Lung_Opacity-610.png,True,False,299,299,L,164.33,07dbea70957f2a02ddd07d53ffb03f4c
Lung_Opacity-611.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-611.png,Lung_Opacity/images/Lung_Opacity-611.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-611.png,Lung_Opacity/masks/Lung_Opacity-611.png,True,False,299,299,L,113.69,2a4ba882bd01b72f7427930440cf821a
Lung_Opacity-612.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-612.png,Lung_Opacity/images/Lung_Opacity-612.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-612.png,Lung_Opacity/masks/Lung_Opacity-612.png,True,False,299,299,L,109.99,ba700019357738fe736c618bce13322f
Lung_Opacity-613.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-613.png,Lung_Opacity/images/Lung_Opacity-613.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-613.png,Lung_Opacity/masks/Lung_Opacity-613.png,True,False,299,299,L,115.54,02c8632c5d995f837ccdf0366fd975e1
Lung_Opacity-614.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-614.png,Lung_Opacity/images/Lung_Opacity-614.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-614.png,Lung_Opacity/masks/Lung_Opacity-614.png,True,False,299,299,L,136.85,a6392d122a9869c4e93f523e2f7adeee
Lung_Opacity-615.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-615.png,Lung_Opacity/images/Lung_Opacity-615.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-615.png,Lung_Opacity/masks/Lung_Opacity-615.png,True,False,299,299,L,137.28,6e4932dbe4f522ceaf7aefc469bf36c5
Lung_Opacity-616.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-616.png,Lung_Opacity/images/Lung_Opacity-616.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-616.png,Lung_Opacity/masks/Lung_Opacity-616.png,True,False,299,299,L,175.16,100ca4838877b559b7e414f362918fa5
Lung_Opacity-617.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-617.png,Lung_Opacity/images/Lung_Opacity-617.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-617.png,Lung_Opacity/masks/Lung_Opacity-617.png,True,False,299,299,L,176.53,d881281eeaf150b24bfd4ed2d7ffeff5
Lung_Opacity-618.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-618.png,Lung_Opacity/images/Lung_Opacity-618.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-618.png,Lung_Opacity/masks/Lung_Opacity-618.png,True,False,299,299,L,118.39,639d34ac6dd52f58ed3129617e1e2dbd
Lung_Opacity-619.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-619.png,Lung_Opacity/images/Lung_Opacity-619.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-619.png,Lung_Opacity/masks/Lung_Opacity-619.png,True,False,299,299,L,160.36,021a41049d398132bc5cac8c885adac5
Lung_Opacity-62.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-62.png,Lung_Opacity/images/Lung_Opacity-62.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-62.png,Lung_Opacity/masks/Lung_Opacity-62.png,True,False,299,299,L,106.76,8371c878b7e8f20b8f2f712b6ad2c733
Lung_Opacity-620.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-620.png,Lung_Opacity/images/Lung_Opacity-620.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-620.png,Lung_Opacity/masks/Lung_Opacity-620.png,True,False,299,299,L,110.85,96e01ff8752fe0efd8ae4e8080939b50
Lung_Opacity-621.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-621.png,Lung_Opacity/images/Lung_Opacity-621.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-621.png,Lung_Opacity/masks/Lung_Opacity-621.png,True,False,299,299,L,158.61,f2d78ca735e3f7d0fd806ee0591c5554
Lung_Opacity-622.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-622.png,Lung_Opacity/images/Lung_Opacity-622.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-622.png,Lung_Opacity/masks/Lung_Opacity-622.png,True,False,299,299,L,139.0,c7af6165fbb81c9ba18f38af7b0f0d80
Lung_Opacity-623.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-623.png,Lung_Opacity/images/Lung_Opacity-623.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-623.png,Lung_Opacity/masks/Lung_Opacity-623.png,True,False,299,299,L,112.79,ab77bd5dbcc419631a277d3e22a28da0
Lung_Opacity-624.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-624.png,Lung_Opacity/images/Lung_Opacity-624.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-624.png,Lung_Opacity/masks/Lung_Opacity-624.png,True,False,299,299,L,133.7,34f90747677ed4fec47c964c6cba5f6f
Lung_Opacity-625.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-625.png,Lung_Opacity/images/Lung_Opacity-625.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-625.png,Lung_Opacity/masks/Lung_Opacity-625.png,True,False,299,299,L,51.16,13f030e9998785d49c6ba39d7baf7976
Lung_Opacity-626.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-626.png,Lung_Opacity/images/Lung_Opacity-626.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-626.png,Lung_Opacity/masks/Lung_Opacity-626.png,True,False,299,299,L,113.7,fbbd200aa9c3ae27c4a93341d444024d
Lung_Opacity-627.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-627.png,Lung_Opacity/images/Lung_Opacity-627.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-627.png,Lung_Opacity/masks/Lung_Opacity-627.png,True,False,299,299,L,140.25,6bc63fbb5ef9a42cd189586a3b960d31
Lung_Opacity-628.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-628.png,Lung_Opacity/images/Lung_Opacity-628.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-628.png,Lung_Opacity/masks/Lung_Opacity-628.png,True,False,299,299,L,106.41,aa8c07e9c246eaf9b49f1283f476a576
Lung_Opacity-629.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-629.png,Lung_Opacity/images/Lung_Opacity-629.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-629.png,Lung_Opacity/masks/Lung_Opacity-629.png,True,False,299,299,L,149.14,59010768f5b1a97c335f31305f20b84e
Lung_Opacity-63.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-63.png,Lung_Opacity/images/Lung_Opacity-63.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-63.png,Lung_Opacity/masks/Lung_Opacity-63.png,True,False,299,299,L,127.39,7eb68cd375786fc603a97bfb2e8b999e
Lung_Opacity-630.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-630.png,Lung_Opacity/images/Lung_Opacity-630.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-630.png,Lung_Opacity/masks/Lung_Opacity-630.png,True,False,299,299,L,99.2,f6da17b034fecb0269a2543f8ae63c2c
Lung_Opacity-631.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-631.png,Lung_Opacity/images/Lung_Opacity-631.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-631.png,Lung_Opacity/masks/Lung_Opacity-631.png,True,False,299,299,L,117.52,91ea7ef88f92244012f999eeaa866920
Lung_Opacity-632.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-632.png,Lung_Opacity/images/Lung_Opacity-632.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-632.png,Lung_Opacity/masks/Lung_Opacity-632.png,True,False,299,299,L,97.24,5d3cafb22aee3f0f411a2bee175c5cf1
Lung_Opacity-633.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-633.png,Lung_Opacity/images/Lung_Opacity-633.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-633.png,Lung_Opacity/masks/Lung_Opacity-633.png,True,False,299,299,L,99.1,dac0f24933e3bbd657ab682223c2e4eb
Lung_Opacity-634.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-634.png,Lung_Opacity/images/Lung_Opacity-634.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-634.png,Lung_Opacity/masks/Lung_Opacity-634.png,True,False,299,299,L,107.2,24cd3197bad065bc557c09eb62b9013b
Lung_Opacity-635.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-635.png,Lung_Opacity/images/Lung_Opacity-635.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-635.png,Lung_Opacity/masks/Lung_Opacity-635.png,True,False,299,299,L,183.8,c0236e28eaa3863a831fd4f987b39044
Lung_Opacity-636.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-636.png,Lung_Opacity/images/Lung_Opacity-636.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-636.png,Lung_Opacity/masks/Lung_Opacity-636.png,True,False,299,299,L,103.08,e7624e8fe78c143559e0a4a3e9642516
Lung_Opacity-637.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-637.png,Lung_Opacity/images/Lung_Opacity-637.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-637.png,Lung_Opacity/masks/Lung_Opacity-637.png,True,False,299,299,L,162.84,61277bcecf6358ce4f6cb5d6c0af7ac1
Lung_Opacity-638.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-638.png,Lung_Opacity/images/Lung_Opacity-638.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-638.png,Lung_Opacity/masks/Lung_Opacity-638.png,True,False,299,299,L,138.78,96ca0b4fb89c75073a2f6c12d65a8cee
Lung_Opacity-639.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-639.png,Lung_Opacity/images/Lung_Opacity-639.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-639.png,Lung_Opacity/masks/Lung_Opacity-639.png,True,False,299,299,L,96.81,ff5227f0b0eb2a7a5eca94bc2df651e7
Lung_Opacity-64.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-64.png,Lung_Opacity/images/Lung_Opacity-64.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-64.png,Lung_Opacity/masks/Lung_Opacity-64.png,True,False,299,299,L,103.35,b71e1fa7fc48c1f8d099695143eb87c0
Lung_Opacity-640.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-640.png,Lung_Opacity/images/Lung_Opacity-640.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-640.png,Lung_Opacity/masks/Lung_Opacity-640.png,True,False,299,299,L,107.59,ed3b9f1dfa31ed55c7fd569b3fd13bc6
Lung_Opacity-641.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-641.png,Lung_Opacity/images/Lung_Opacity-641.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-641.png,Lung_Opacity/masks/Lung_Opacity-641.png,True,False,299,299,L,130.55,f2cfdf6e6aefd1b66c838c3bd99560fc
Lung_Opacity-642.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-642.png,Lung_Opacity/images/Lung_Opacity-642.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-642.png,Lung_Opacity/masks/Lung_Opacity-642.png,True,False,299,299,L,143.43,7936c4cc7835b37f30765797cb7f4dba
Lung_Opacity-643.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-643.png,Lung_Opacity/images/Lung_Opacity-643.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-643.png,Lung_Opacity/masks/Lung_Opacity-643.png,True,False,299,299,L,105.94,62389f30be1537300fd45cc50b1d7f7f
Lung_Opacity-644.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-644.png,Lung_Opacity/images/Lung_Opacity-644.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-644.png,Lung_Opacity/masks/Lung_Opacity-644.png,True,False,299,299,L,117.02,65b84d03bef2d49d1d3fe07029375a0c
Lung_Opacity-645.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-645.png,Lung_Opacity/images/Lung_Opacity-645.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-645.png,Lung_Opacity/masks/Lung_Opacity-645.png,True,False,299,299,L,178.74,9d967b985895a5d854a7355823682aa6
Lung_Opacity-646.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-646.png,Lung_Opacity/images/Lung_Opacity-646.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-646.png,Lung_Opacity/masks/Lung_Opacity-646.png,True,False,299,299,L,165.3,fd58b7c3f807297c092ecb9e49304c83
Lung_Opacity-647.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-647.png,Lung_Opacity/images/Lung_Opacity-647.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-647.png,Lung_Opacity/masks/Lung_Opacity-647.png,True,False,299,299,L,111.29,c5a3fa1d4690d4086f7913c8d9f2cdd6
Lung_Opacity-648.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-648.png,Lung_Opacity/images/Lung_Opacity-648.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-648.png,Lung_Opacity/masks/Lung_Opacity-648.png,True,False,299,299,L,115.76,723c12e1abcbe9e2dbd70cd85da0a378
Lung_Opacity-649.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-649.png,Lung_Opacity/images/Lung_Opacity-649.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-649.png,Lung_Opacity/masks/Lung_Opacity-649.png,True,False,299,299,L,114.29,9e52586c501ffb2268f6104ce9f8d6e7
Lung_Opacity-65.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-65.png,Lung_Opacity/images/Lung_Opacity-65.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-65.png,Lung_Opacity/masks/Lung_Opacity-65.png,True,False,299,299,L,130.17,30f55233ad0a41bd9459573f61b3388b
Lung_Opacity-650.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-650.png,Lung_Opacity/images/Lung_Opacity-650.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-650.png,Lung_Opacity/masks/Lung_Opacity-650.png,True,False,299,299,L,131.47,4ebe850977806e0b1574f653aefcfa2c
Lung_Opacity-651.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-651.png,Lung_Opacity/images/Lung_Opacity-651.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-651.png,Lung_Opacity/masks/Lung_Opacity-651.png,True,False,299,299,L,168.12,74d5dee6ba37630fc3b41cc2c47eae4c
Lung_Opacity-652.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-652.png,Lung_Opacity/images/Lung_Opacity-652.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-652.png,Lung_Opacity/masks/Lung_Opacity-652.png,True,False,299,299,L,113.84,ee25f82d0f58f4ee8f3a5eb71b65fe83
Lung_Opacity-653.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-653.png,Lung_Opacity/images/Lung_Opacity-653.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-653.png,Lung_Opacity/masks/Lung_Opacity-653.png,True,False,299,299,L,114.05,1879d8388941edfe3a5c5c62af6f1294
Lung_Opacity-654.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-654.png,Lung_Opacity/images/Lung_Opacity-654.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-654.png,Lung_Opacity/masks/Lung_Opacity-654.png,True,False,299,299,L,122.01,cbf113549516670435fafe5f7f55848d
Lung_Opacity-655.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-655.png,Lung_Opacity/images/Lung_Opacity-655.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-655.png,Lung_Opacity/masks/Lung_Opacity-655.png,True,False,299,299,L,148.52,be1b4b27069c1fbbf2d0da4a80d45453
Lung_Opacity-656.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-656.png,Lung_Opacity/images/Lung_Opacity-656.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-656.png,Lung_Opacity/masks/Lung_Opacity-656.png,True,False,299,299,L,169.07,6d4ba442a9797ed5d99697069e4204f5
Lung_Opacity-657.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-657.png,Lung_Opacity/images/Lung_Opacity-657.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-657.png,Lung_Opacity/masks/Lung_Opacity-657.png,True,False,299,299,L,116.07,3abb124edb3b8d5069d4ee35a1f5167c
Lung_Opacity-658.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-658.png,Lung_Opacity/images/Lung_Opacity-658.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-658.png,Lung_Opacity/masks/Lung_Opacity-658.png,True,False,299,299,L,117.56,60dfcf329df1cb89f070e097396c3ea3
Lung_Opacity-659.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-659.png,Lung_Opacity/images/Lung_Opacity-659.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-659.png,Lung_Opacity/masks/Lung_Opacity-659.png,True,False,299,299,L,136.55,bd142621d3d5803a9d54e82258ceeea1
Lung_Opacity-66.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-66.png,Lung_Opacity/images/Lung_Opacity-66.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-66.png,Lung_Opacity/masks/Lung_Opacity-66.png,True,False,299,299,L,140.77,6e530d789d785e2dd51f2dd10afa9931
Lung_Opacity-660.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-660.png,Lung_Opacity/images/Lung_Opacity-660.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-660.png,Lung_Opacity/masks/Lung_Opacity-660.png,True,False,299,299,L,134.76,a6133c98380236d8c8c6626597ba815c
Lung_Opacity-661.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-661.png,Lung_Opacity/images/Lung_Opacity-661.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-661.png,Lung_Opacity/masks/Lung_Opacity-661.png,True,False,299,299,L,106.7,2f56a0682297f442d3e8aa4c68c6e65f
Lung_Opacity-662.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-662.png,Lung_Opacity/images/Lung_Opacity-662.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-662.png,Lung_Opacity/masks/Lung_Opacity-662.png,True,False,299,299,L,156.86,97abdae3a15d14a22405d6ce59865a9b
Lung_Opacity-663.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-663.png,Lung_Opacity/images/Lung_Opacity-663.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-663.png,Lung_Opacity/masks/Lung_Opacity-663.png,True,False,299,299,L,110.83,d4ad93abf0c75f00abc3f31e927d1c03
Lung_Opacity-664.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-664.png,Lung_Opacity/images/Lung_Opacity-664.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-664.png,Lung_Opacity/masks/Lung_Opacity-664.png,True,False,299,299,L,164.19,fc36822c36d56ef330469c3d0d91c3da
Lung_Opacity-665.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-665.png,Lung_Opacity/images/Lung_Opacity-665.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-665.png,Lung_Opacity/masks/Lung_Opacity-665.png,True,False,299,299,L,115.06,9c203f6c677536970bdd8a6c5bca48b0
Lung_Opacity-666.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-666.png,Lung_Opacity/images/Lung_Opacity-666.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-666.png,Lung_Opacity/masks/Lung_Opacity-666.png,True,False,299,299,L,89.73,429ab8307bd04ee7717994111dde8536
Lung_Opacity-667.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-667.png,Lung_Opacity/images/Lung_Opacity-667.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-667.png,Lung_Opacity/masks/Lung_Opacity-667.png,True,False,299,299,L,119.02,f52721e2d6f9425f5c95b22b0c9478d4
Lung_Opacity-668.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-668.png,Lung_Opacity/images/Lung_Opacity-668.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-668.png,Lung_Opacity/masks/Lung_Opacity-668.png,True,False,299,299,L,126.37,6bd115e7933debdd058228ae2447cf07
Lung_Opacity-669.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-669.png,Lung_Opacity/images/Lung_Opacity-669.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-669.png,Lung_Opacity/masks/Lung_Opacity-669.png,True,False,299,299,L,141.7,b8f56f9ae61941b4dd6a48ac8da79bae
Lung_Opacity-67.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-67.png,Lung_Opacity/images/Lung_Opacity-67.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-67.png,Lung_Opacity/masks/Lung_Opacity-67.png,True,False,299,299,L,130.79,1cec58e6718873a139c75bc42f0dd18a
Lung_Opacity-670.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-670.png,Lung_Opacity/images/Lung_Opacity-670.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-670.png,Lung_Opacity/masks/Lung_Opacity-670.png,True,False,299,299,L,106.76,3528651508c717027685e979ccf3b47f
Lung_Opacity-671.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-671.png,Lung_Opacity/images/Lung_Opacity-671.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-671.png,Lung_Opacity/masks/Lung_Opacity-671.png,True,False,299,299,L,120.76,fa46c0158975942616b3b748c46a6af8
Lung_Opacity-672.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-672.png,Lung_Opacity/images/Lung_Opacity-672.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-672.png,Lung_Opacity/masks/Lung_Opacity-672.png,True,False,299,299,L,114.91,0a4363fde0d96ea62486d1645acf4aa1
Lung_Opacity-673.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-673.png,Lung_Opacity/images/Lung_Opacity-673.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-673.png,Lung_Opacity/masks/Lung_Opacity-673.png,True,False,299,299,L,124.91,5f13fdfa71b069edf333d73b723747eb
Lung_Opacity-674.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-674.png,Lung_Opacity/images/Lung_Opacity-674.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-674.png,Lung_Opacity/masks/Lung_Opacity-674.png,True,False,299,299,L,128.14,5e6c0d682d6358876407d85e4b8a66e7
Lung_Opacity-675.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-675.png,Lung_Opacity/images/Lung_Opacity-675.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-675.png,Lung_Opacity/masks/Lung_Opacity-675.png,True,False,299,299,L,98.33,015df503b0020214ad186e9d5824bf82
Lung_Opacity-676.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-676.png,Lung_Opacity/images/Lung_Opacity-676.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-676.png,Lung_Opacity/masks/Lung_Opacity-676.png,True,False,299,299,L,110.33,a4d1fffe690720b44d75829e19b804dc
Lung_Opacity-677.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-677.png,Lung_Opacity/images/Lung_Opacity-677.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-677.png,Lung_Opacity/masks/Lung_Opacity-677.png,True,False,299,299,L,134.61,cd0ce597a17b1b086ff080c0ba2713ad
Lung_Opacity-678.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-678.png,Lung_Opacity/images/Lung_Opacity-678.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-678.png,Lung_Opacity/masks/Lung_Opacity-678.png,True,False,299,299,L,127.54,75301e12cf980776c570aaf97b1e7624
Lung_Opacity-679.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-679.png,Lung_Opacity/images/Lung_Opacity-679.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-679.png,Lung_Opacity/masks/Lung_Opacity-679.png,True,False,299,299,L,158.03,bbbb57817ded89a9f5d981856fa25d41
Lung_Opacity-68.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-68.png,Lung_Opacity/images/Lung_Opacity-68.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-68.png,Lung_Opacity/masks/Lung_Opacity-68.png,True,False,299,299,L,166.34,30120f42634a7e22befc683a14afe7f3
Lung_Opacity-680.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-680.png,Lung_Opacity/images/Lung_Opacity-680.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-680.png,Lung_Opacity/masks/Lung_Opacity-680.png,True,False,299,299,L,110.48,47c4004fe1ce16189769f25475cf3378
Lung_Opacity-681.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-681.png,Lung_Opacity/images/Lung_Opacity-681.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-681.png,Lung_Opacity/masks/Lung_Opacity-681.png,True,False,299,299,L,110.98,b508d2840dc74d2f5102f40ae5a32911
Lung_Opacity-682.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-682.png,Lung_Opacity/images/Lung_Opacity-682.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-682.png,Lung_Opacity/masks/Lung_Opacity-682.png,True,False,299,299,L,128.0,5fca87f7779fae1f309b907501ceec88
Lung_Opacity-683.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-683.png,Lung_Opacity/images/Lung_Opacity-683.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-683.png,Lung_Opacity/masks/Lung_Opacity-683.png,True,False,299,299,L,175.8,3a8e3b22804b2b99068267e1089e2327
Lung_Opacity-684.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-684.png,Lung_Opacity/images/Lung_Opacity-684.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-684.png,Lung_Opacity/masks/Lung_Opacity-684.png,True,False,299,299,L,164.41,3088991d252ec91caad8e4755266e607
Lung_Opacity-685.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-685.png,Lung_Opacity/images/Lung_Opacity-685.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-685.png,Lung_Opacity/masks/Lung_Opacity-685.png,True,False,299,299,L,115.87,88023487da81e02edea1569ee2380007
Lung_Opacity-686.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-686.png,Lung_Opacity/images/Lung_Opacity-686.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-686.png,Lung_Opacity/masks/Lung_Opacity-686.png,True,False,299,299,L,106.87,22f84ee59ca38c4d5de6fd7c8781d5a5
Lung_Opacity-687.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-687.png,Lung_Opacity/images/Lung_Opacity-687.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-687.png,Lung_Opacity/masks/Lung_Opacity-687.png,True,False,299,299,L,114.51,aecc36c0c4f002f064800563b94d3965
Lung_Opacity-688.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-688.png,Lung_Opacity/images/Lung_Opacity-688.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-688.png,Lung_Opacity/masks/Lung_Opacity-688.png,True,False,299,299,L,109.76,cb039039acb2ab2f46a01671ac27312f
Lung_Opacity-689.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-689.png,Lung_Opacity/images/Lung_Opacity-689.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-689.png,Lung_Opacity/masks/Lung_Opacity-689.png,True,False,299,299,L,108.37,dafd0f6e710323c6e2826fcb31081da0
Lung_Opacity-69.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-69.png,Lung_Opacity/images/Lung_Opacity-69.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-69.png,Lung_Opacity/masks/Lung_Opacity-69.png,True,False,299,299,L,131.15,85c5ba6c831c5d98cbcfa2e7ed248ea6
Lung_Opacity-690.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-690.png,Lung_Opacity/images/Lung_Opacity-690.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-690.png,Lung_Opacity/masks/Lung_Opacity-690.png,True,False,299,299,L,178.66,9c86233f11aa1046507b12d4c4bc9ea4
Lung_Opacity-691.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-691.png,Lung_Opacity/images/Lung_Opacity-691.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-691.png,Lung_Opacity/masks/Lung_Opacity-691.png,True,False,299,299,L,135.74,4707a25797d9de2adb60e7d79b629dec
Lung_Opacity-692.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-692.png,Lung_Opacity/images/Lung_Opacity-692.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-692.png,Lung_Opacity/masks/Lung_Opacity-692.png,True,False,299,299,L,140.95,d245d890bf5dded61fb181c38c59d84d
Lung_Opacity-693.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-693.png,Lung_Opacity/images/Lung_Opacity-693.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-693.png,Lung_Opacity/masks/Lung_Opacity-693.png,True,False,299,299,L,106.27,37beb1e43ae66040b68fd85f6265082c
Lung_Opacity-694.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-694.png,Lung_Opacity/images/Lung_Opacity-694.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-694.png,Lung_Opacity/masks/Lung_Opacity-694.png,True,False,299,299,L,107.78,6fce988b829e3401c6f701000eec563e
Lung_Opacity-695.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-695.png,Lung_Opacity/images/Lung_Opacity-695.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-695.png,Lung_Opacity/masks/Lung_Opacity-695.png,True,False,299,299,L,117.35,d9a2d5777855c42c8dae9790b57e4300
Lung_Opacity-696.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-696.png,Lung_Opacity/images/Lung_Opacity-696.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-696.png,Lung_Opacity/masks/Lung_Opacity-696.png,True,False,299,299,L,148.42,c85b87f3abe6ce482f3270d31e94be18
Lung_Opacity-697.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-697.png,Lung_Opacity/images/Lung_Opacity-697.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-697.png,Lung_Opacity/masks/Lung_Opacity-697.png,True,False,299,299,L,111.46,8f7c407faa5f2521270b648638cd8ee9
Lung_Opacity-698.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-698.png,Lung_Opacity/images/Lung_Opacity-698.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-698.png,Lung_Opacity/masks/Lung_Opacity-698.png,True,False,299,299,L,146.93,3248c2400d581fbe86b087fac1b61cc6
Lung_Opacity-699.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-699.png,Lung_Opacity/images/Lung_Opacity-699.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-699.png,Lung_Opacity/masks/Lung_Opacity-699.png,True,False,299,299,L,99.94,bdb58f1a96b7f6d7ad780e655679b96b
Lung_Opacity-7.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-7.png,Lung_Opacity/images/Lung_Opacity-7.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-7.png,Lung_Opacity/masks/Lung_Opacity-7.png,True,False,299,299,L,152.46,64f4aa1f38afdb4ca6349a1ee5d13358
Lung_Opacity-70.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-70.png,Lung_Opacity/images/Lung_Opacity-70.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-70.png,Lung_Opacity/masks/Lung_Opacity-70.png,True,False,299,299,L,121.93,98c72f4ebbb102411b071d38809bca2f
Lung_Opacity-700.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-700.png,Lung_Opacity/images/Lung_Opacity-700.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-700.png,Lung_Opacity/masks/Lung_Opacity-700.png,True,False,299,299,L,120.22,e0d6f30bd9afc7589466894625e5e711
Lung_Opacity-701.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-701.png,Lung_Opacity/images/Lung_Opacity-701.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-701.png,Lung_Opacity/masks/Lung_Opacity-701.png,True,False,299,299,L,97.78,fdb6223c4f25c1aafc02912d355e636e
Lung_Opacity-702.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-702.png,Lung_Opacity/images/Lung_Opacity-702.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-702.png,Lung_Opacity/masks/Lung_Opacity-702.png,True,False,299,299,L,142.98,943014d510149d15d5147c105fe01b03
Lung_Opacity-703.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-703.png,Lung_Opacity/images/Lung_Opacity-703.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-703.png,Lung_Opacity/masks/Lung_Opacity-703.png,True,False,299,299,L,165.44,34ae47034a1b0f1339f985218d0bff3a
Lung_Opacity-704.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-704.png,Lung_Opacity/images/Lung_Opacity-704.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-704.png,Lung_Opacity/masks/Lung_Opacity-704.png,True,False,299,299,L,116.56,2edf26a16904be75aed932bba92e64cf
Lung_Opacity-705.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-705.png,Lung_Opacity/images/Lung_Opacity-705.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-705.png,Lung_Opacity/masks/Lung_Opacity-705.png,True,False,299,299,L,127.27,bd0e80b076f8436f09e3d3df3c2e9f09
Lung_Opacity-706.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-706.png,Lung_Opacity/images/Lung_Opacity-706.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-706.png,Lung_Opacity/masks/Lung_Opacity-706.png,True,False,299,299,L,105.08,b6b48115536b49ec2d72d23811a3909a
Lung_Opacity-707.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-707.png,Lung_Opacity/images/Lung_Opacity-707.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-707.png,Lung_Opacity/masks/Lung_Opacity-707.png,True,False,299,299,L,132.48,1efaddb9d243d6dc5f75c2de51eee485
Lung_Opacity-708.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-708.png,Lung_Opacity/images/Lung_Opacity-708.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-708.png,Lung_Opacity/masks/Lung_Opacity-708.png,True,False,299,299,L,123.38,cdb9f44f077a5c78a736bf8c227841d1
Lung_Opacity-709.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-709.png,Lung_Opacity/images/Lung_Opacity-709.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-709.png,Lung_Opacity/masks/Lung_Opacity-709.png,True,False,299,299,L,144.42,283d215aa361276e2c664e11123039da
Lung_Opacity-71.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-71.png,Lung_Opacity/images/Lung_Opacity-71.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-71.png,Lung_Opacity/masks/Lung_Opacity-71.png,True,False,299,299,L,129.44,e91f61d14171ece580a0cebf37d5c683
Lung_Opacity-710.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-710.png,Lung_Opacity/images/Lung_Opacity-710.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-710.png,Lung_Opacity/masks/Lung_Opacity-710.png,True,False,299,299,L,108.51,3aba26e45670b95ed555d60afe921fa1
Lung_Opacity-711.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-711.png,Lung_Opacity/images/Lung_Opacity-711.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-711.png,Lung_Opacity/masks/Lung_Opacity-711.png,True,False,299,299,L,83.48,cc04244e388a118cc906f70454079938
Lung_Opacity-712.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-712.png,Lung_Opacity/images/Lung_Opacity-712.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-712.png,Lung_Opacity/masks/Lung_Opacity-712.png,True,False,299,299,L,129.6,2d163e5135d1489569909a63249ba435
Lung_Opacity-713.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-713.png,Lung_Opacity/images/Lung_Opacity-713.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-713.png,Lung_Opacity/masks/Lung_Opacity-713.png,True,False,299,299,L,137.54,b076d93feb86821f94dc65b3238e6af9
Lung_Opacity-714.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-714.png,Lung_Opacity/images/Lung_Opacity-714.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-714.png,Lung_Opacity/masks/Lung_Opacity-714.png,True,False,299,299,L,114.37,e6da787f74b4b75c3cdf0d1319f154ff
Lung_Opacity-715.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-715.png,Lung_Opacity/images/Lung_Opacity-715.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-715.png,Lung_Opacity/masks/Lung_Opacity-715.png,True,False,299,299,L,118.33,807176874f3f12fbd9d21bb195a220d4
Lung_Opacity-716.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-716.png,Lung_Opacity/images/Lung_Opacity-716.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-716.png,Lung_Opacity/masks/Lung_Opacity-716.png,True,False,299,299,L,131.05,20f0422352cd01bde793c622a23f93d6
Lung_Opacity-717.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-717.png,Lung_Opacity/images/Lung_Opacity-717.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-717.png,Lung_Opacity/masks/Lung_Opacity-717.png,True,False,299,299,L,97.5,f051425114d39889255b4060b73062a7
Lung_Opacity-718.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-718.png,Lung_Opacity/images/Lung_Opacity-718.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-718.png,Lung_Opacity/masks/Lung_Opacity-718.png,True,False,299,299,L,106.19,b02a4d58f15b34c4bc2d1a3da39339d3
Lung_Opacity-719.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-719.png,Lung_Opacity/images/Lung_Opacity-719.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-719.png,Lung_Opacity/masks/Lung_Opacity-719.png,True,False,299,299,L,121.32,8afbf40474f3bd28a6b6823d89586b25
Lung_Opacity-72.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-72.png,Lung_Opacity/images/Lung_Opacity-72.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-72.png,Lung_Opacity/masks/Lung_Opacity-72.png,True,False,299,299,L,133.19,704eabb29adb262e26a8cb01dafcf564
Lung_Opacity-720.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-720.png,Lung_Opacity/images/Lung_Opacity-720.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-720.png,Lung_Opacity/masks/Lung_Opacity-720.png,True,False,299,299,L,177.33,3a5a65490e1b64429066da0d13ba1f2f
Lung_Opacity-721.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-721.png,Lung_Opacity/images/Lung_Opacity-721.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-721.png,Lung_Opacity/masks/Lung_Opacity-721.png,True,False,299,299,L,156.16,937997bf7d65b0cdb16fcfaabab8f611
Lung_Opacity-722.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-722.png,Lung_Opacity/images/Lung_Opacity-722.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-722.png,Lung_Opacity/masks/Lung_Opacity-722.png,True,False,299,299,L,115.85,c46b108e1d1d1fb877c0d9b7eee695c1
Lung_Opacity-723.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-723.png,Lung_Opacity/images/Lung_Opacity-723.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-723.png,Lung_Opacity/masks/Lung_Opacity-723.png,True,False,299,299,L,169.76,0d744c4ffccde1c9b1e07c7f6f1c0cc0
Lung_Opacity-724.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-724.png,Lung_Opacity/images/Lung_Opacity-724.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-724.png,Lung_Opacity/masks/Lung_Opacity-724.png,True,False,299,299,L,148.81,d0e35bc19308ac1360d061c36ca2dddc
Lung_Opacity-725.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-725.png,Lung_Opacity/images/Lung_Opacity-725.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-725.png,Lung_Opacity/masks/Lung_Opacity-725.png,True,False,299,299,L,118.64,b6c08e85db213e7de415c528c7e1a5a5
Lung_Opacity-726.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-726.png,Lung_Opacity/images/Lung_Opacity-726.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-726.png,Lung_Opacity/masks/Lung_Opacity-726.png,True,False,299,299,L,138.29,a6043bdb9a852e2806e3962e653ce384
Lung_Opacity-727.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-727.png,Lung_Opacity/images/Lung_Opacity-727.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-727.png,Lung_Opacity/masks/Lung_Opacity-727.png,True,False,299,299,L,149.49,a6acb079dad720bd23b62351a1b348cf
Lung_Opacity-728.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-728.png,Lung_Opacity/images/Lung_Opacity-728.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-728.png,Lung_Opacity/masks/Lung_Opacity-728.png,True,False,299,299,L,167.12,a51a21ea9c542c759f90863d93af0a8f
Lung_Opacity-729.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-729.png,Lung_Opacity/images/Lung_Opacity-729.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-729.png,Lung_Opacity/masks/Lung_Opacity-729.png,True,False,299,299,L,121.23,5d7bea7d54708952ab5f034d726f34da
Lung_Opacity-73.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-73.png,Lung_Opacity/images/Lung_Opacity-73.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-73.png,Lung_Opacity/masks/Lung_Opacity-73.png,True,False,299,299,L,135.36,24eb19715b617225d445808732241dd8
Lung_Opacity-730.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-730.png,Lung_Opacity/images/Lung_Opacity-730.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-730.png,Lung_Opacity/masks/Lung_Opacity-730.png,True,False,299,299,L,145.72,739c9c0f1c115d07dfad62e8770d1bbb
Lung_Opacity-731.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-731.png,Lung_Opacity/images/Lung_Opacity-731.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-731.png,Lung_Opacity/masks/Lung_Opacity-731.png,True,False,299,299,L,120.68,79f0397ff69856339c23327fc1565a1b
Lung_Opacity-732.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-732.png,Lung_Opacity/images/Lung_Opacity-732.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-732.png,Lung_Opacity/masks/Lung_Opacity-732.png,True,False,299,299,L,139.02,14c79837b8db8005d9e6018902bb6752
Lung_Opacity-733.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-733.png,Lung_Opacity/images/Lung_Opacity-733.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-733.png,Lung_Opacity/masks/Lung_Opacity-733.png,True,False,299,299,L,143.21,31351e541d57082694276c92d16ce1bd
Lung_Opacity-734.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-734.png,Lung_Opacity/images/Lung_Opacity-734.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-734.png,Lung_Opacity/masks/Lung_Opacity-734.png,True,False,299,299,L,167.92,59d4d4abd6f24bbd16a0da4ba8277690
Lung_Opacity-735.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-735.png,Lung_Opacity/images/Lung_Opacity-735.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-735.png,Lung_Opacity/masks/Lung_Opacity-735.png,True,False,299,299,L,114.99,e860a8106e35d7cc169ed02ad8702c8a
Lung_Opacity-736.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-736.png,Lung_Opacity/images/Lung_Opacity-736.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-736.png,Lung_Opacity/masks/Lung_Opacity-736.png,True,False,299,299,L,100.37,7d29cc808ed5f5d92f0c5023779f3221
Lung_Opacity-737.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-737.png,Lung_Opacity/images/Lung_Opacity-737.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-737.png,Lung_Opacity/masks/Lung_Opacity-737.png,True,False,299,299,L,144.99,f14b979fba65c29359fc7e9b6b04c6e2
Lung_Opacity-738.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-738.png,Lung_Opacity/images/Lung_Opacity-738.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-738.png,Lung_Opacity/masks/Lung_Opacity-738.png,True,False,299,299,L,154.58,d626797b6bce7d111b082926eb17bcd0
Lung_Opacity-739.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-739.png,Lung_Opacity/images/Lung_Opacity-739.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-739.png,Lung_Opacity/masks/Lung_Opacity-739.png,True,False,299,299,L,114.6,07de50d990311090eae91f79ed6df8bb
Lung_Opacity-74.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-74.png,Lung_Opacity/images/Lung_Opacity-74.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-74.png,Lung_Opacity/masks/Lung_Opacity-74.png,True,False,299,299,L,116.74,dab9a26b36d15eceb214340217b43304
Lung_Opacity-740.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-740.png,Lung_Opacity/images/Lung_Opacity-740.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-740.png,Lung_Opacity/masks/Lung_Opacity-740.png,True,False,299,299,L,113.03,f17d824be7b2133f179b994050b54ab7
Lung_Opacity-741.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-741.png,Lung_Opacity/images/Lung_Opacity-741.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-741.png,Lung_Opacity/masks/Lung_Opacity-741.png,True,False,299,299,L,96.96,771f6e884a263d4ed13f0d480d8ef3fa
Lung_Opacity-742.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-742.png,Lung_Opacity/images/Lung_Opacity-742.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-742.png,Lung_Opacity/masks/Lung_Opacity-742.png,True,False,299,299,L,96.74,bbd93efbdf52c9fe805ae70892ef30d6
Lung_Opacity-743.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-743.png,Lung_Opacity/images/Lung_Opacity-743.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-743.png,Lung_Opacity/masks/Lung_Opacity-743.png,True,False,299,299,L,108.26,8ced7dc932e4ff2433ff2f14d2b337fe
Lung_Opacity-744.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-744.png,Lung_Opacity/images/Lung_Opacity-744.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-744.png,Lung_Opacity/masks/Lung_Opacity-744.png,True,False,299,299,L,147.79,9bcba1f5d66db0b2d48d381f2b925b33
Lung_Opacity-745.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-745.png,Lung_Opacity/images/Lung_Opacity-745.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-745.png,Lung_Opacity/masks/Lung_Opacity-745.png,True,False,299,299,L,146.76,0a6d6683161005a3bb312d6c9055963e
Lung_Opacity-746.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-746.png,Lung_Opacity/images/Lung_Opacity-746.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-746.png,Lung_Opacity/masks/Lung_Opacity-746.png,True,False,299,299,L,109.57,fd9f5a0eff8246a0d66225e1d0027dcf
Lung_Opacity-747.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-747.png,Lung_Opacity/images/Lung_Opacity-747.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-747.png,Lung_Opacity/masks/Lung_Opacity-747.png,True,False,299,299,L,106.85,8be471feb73ae92acc730471e0a4a081
Lung_Opacity-748.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-748.png,Lung_Opacity/images/Lung_Opacity-748.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-748.png,Lung_Opacity/masks/Lung_Opacity-748.png,True,False,299,299,L,116.77,5fb90307472692f33ae3cb9fecd315c8
Lung_Opacity-749.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-749.png,Lung_Opacity/images/Lung_Opacity-749.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-749.png,Lung_Opacity/masks/Lung_Opacity-749.png,True,False,299,299,L,118.14,3b41f27fdad54d6bada2059898c77b99
Lung_Opacity-75.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-75.png,Lung_Opacity/images/Lung_Opacity-75.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-75.png,Lung_Opacity/masks/Lung_Opacity-75.png,True,False,299,299,L,117.07,a3d56d580df3e8becce237d78e8a5bab
Lung_Opacity-750.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-750.png,Lung_Opacity/images/Lung_Opacity-750.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-750.png,Lung_Opacity/masks/Lung_Opacity-750.png,True,False,299,299,L,122.45,0360289f16631be67b135d461ef63159
Lung_Opacity-751.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-751.png,Lung_Opacity/images/Lung_Opacity-751.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-751.png,Lung_Opacity/masks/Lung_Opacity-751.png,True,False,299,299,L,98.39,d1e2866bb5fcb4b4e3d0f43f8b154b07
Lung_Opacity-752.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-752.png,Lung_Opacity/images/Lung_Opacity-752.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-752.png,Lung_Opacity/masks/Lung_Opacity-752.png,True,False,299,299,L,108.56,acfb2dbfa134f8b74cb2b948cf03f43c
Lung_Opacity-753.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-753.png,Lung_Opacity/images/Lung_Opacity-753.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-753.png,Lung_Opacity/masks/Lung_Opacity-753.png,True,False,299,299,L,117.27,10111e39086a6ada05cd73f341a9d193
Lung_Opacity-754.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-754.png,Lung_Opacity/images/Lung_Opacity-754.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-754.png,Lung_Opacity/masks/Lung_Opacity-754.png,True,False,299,299,L,88.95,49c608e519b04d654d260b2ff3844a1e
Lung_Opacity-755.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-755.png,Lung_Opacity/images/Lung_Opacity-755.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-755.png,Lung_Opacity/masks/Lung_Opacity-755.png,True,False,299,299,L,62.06,207831dd39da6c492de9b7f8c60cf86e
Lung_Opacity-756.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-756.png,Lung_Opacity/images/Lung_Opacity-756.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-756.png,Lung_Opacity/masks/Lung_Opacity-756.png,True,False,299,299,L,118.81,1476a435ef64eec2a7c246b4dd48303c
Lung_Opacity-757.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-757.png,Lung_Opacity/images/Lung_Opacity-757.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-757.png,Lung_Opacity/masks/Lung_Opacity-757.png,True,False,299,299,L,107.66,a02e0b9214bc5dcb2e521a663af9274a
Lung_Opacity-758.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-758.png,Lung_Opacity/images/Lung_Opacity-758.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-758.png,Lung_Opacity/masks/Lung_Opacity-758.png,True,False,299,299,L,119.39,08721552c37cb1724fbd86cc1cd5b128
Lung_Opacity-759.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-759.png,Lung_Opacity/images/Lung_Opacity-759.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-759.png,Lung_Opacity/masks/Lung_Opacity-759.png,True,False,299,299,L,152.84,b0ab3faf86b1eb330fa33178f4ace0fa
Lung_Opacity-76.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-76.png,Lung_Opacity/images/Lung_Opacity-76.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-76.png,Lung_Opacity/masks/Lung_Opacity-76.png,True,False,299,299,L,140.38,6ace1f15f4f325d5af6572cfb57e7643
Lung_Opacity-760.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-760.png,Lung_Opacity/images/Lung_Opacity-760.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-760.png,Lung_Opacity/masks/Lung_Opacity-760.png,True,False,299,299,L,160.81,8517582c1b5e76e72d2dc6b074a3f82e
Lung_Opacity-761.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-761.png,Lung_Opacity/images/Lung_Opacity-761.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-761.png,Lung_Opacity/masks/Lung_Opacity-761.png,True,False,299,299,L,125.18,991bfbb169426ea48c94391148348e16
Lung_Opacity-762.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-762.png,Lung_Opacity/images/Lung_Opacity-762.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-762.png,Lung_Opacity/masks/Lung_Opacity-762.png,True,False,299,299,L,139.16,7234d6ddfee5f440f3f8a532fc109563
Lung_Opacity-763.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-763.png,Lung_Opacity/images/Lung_Opacity-763.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-763.png,Lung_Opacity/masks/Lung_Opacity-763.png,True,False,299,299,L,137.59,02f4ce1dc8052a7c764a5171e2bafed5
Lung_Opacity-764.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-764.png,Lung_Opacity/images/Lung_Opacity-764.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-764.png,Lung_Opacity/masks/Lung_Opacity-764.png,True,False,299,299,L,136.38,21ae86bcee371db16ad7f127eb6938a2
Lung_Opacity-765.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-765.png,Lung_Opacity/images/Lung_Opacity-765.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-765.png,Lung_Opacity/masks/Lung_Opacity-765.png,True,False,299,299,L,153.13,01cc1bdb239b7554c674c3502743e261
Lung_Opacity-766.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-766.png,Lung_Opacity/images/Lung_Opacity-766.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-766.png,Lung_Opacity/masks/Lung_Opacity-766.png,True,False,299,299,L,103.18,86442db1a74bef6568cac17e5864d452
Lung_Opacity-767.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-767.png,Lung_Opacity/images/Lung_Opacity-767.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-767.png,Lung_Opacity/masks/Lung_Opacity-767.png,True,False,299,299,L,152.16,e5441bff5d59024a0fd944ab3bc57dfe
Lung_Opacity-768.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-768.png,Lung_Opacity/images/Lung_Opacity-768.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-768.png,Lung_Opacity/masks/Lung_Opacity-768.png,True,False,299,299,L,115.06,c917dcc76f1ae91971312c8289aee5f6
Lung_Opacity-769.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-769.png,Lung_Opacity/images/Lung_Opacity-769.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-769.png,Lung_Opacity/masks/Lung_Opacity-769.png,True,False,299,299,L,168.07,ad05fdae80c6fa1c3db8f0864bca525f
Lung_Opacity-77.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-77.png,Lung_Opacity/images/Lung_Opacity-77.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-77.png,Lung_Opacity/masks/Lung_Opacity-77.png,True,False,299,299,L,157.08,e3dd0d6ca5d305e4dcc0d2af474d7998
Lung_Opacity-770.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-770.png,Lung_Opacity/images/Lung_Opacity-770.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-770.png,Lung_Opacity/masks/Lung_Opacity-770.png,True,False,299,299,L,111.24,6b2c3f37f253cfa183506ae26d7b6e8a
Lung_Opacity-771.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-771.png,Lung_Opacity/images/Lung_Opacity-771.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-771.png,Lung_Opacity/masks/Lung_Opacity-771.png,True,False,299,299,L,109.19,326c95103447b4910cf3de479f1a4dc0
Lung_Opacity-772.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-772.png,Lung_Opacity/images/Lung_Opacity-772.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-772.png,Lung_Opacity/masks/Lung_Opacity-772.png,True,False,299,299,L,127.45,41bee777a611b2bcd0f82f852982c681
Lung_Opacity-773.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-773.png,Lung_Opacity/images/Lung_Opacity-773.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-773.png,Lung_Opacity/masks/Lung_Opacity-773.png,True,False,299,299,L,116.28,73f8873b6e9b1633121a025b8bc2afa9
Lung_Opacity-774.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-774.png,Lung_Opacity/images/Lung_Opacity-774.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-774.png,Lung_Opacity/masks/Lung_Opacity-774.png,True,False,299,299,L,105.94,94fc82fe0af3f220cf89d07800bf7ad5
Lung_Opacity-775.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-775.png,Lung_Opacity/images/Lung_Opacity-775.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-775.png,Lung_Opacity/masks/Lung_Opacity-775.png,True,False,299,299,L,109.22,0704e39c73300345882ee581a802e5a7
Lung_Opacity-776.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-776.png,Lung_Opacity/images/Lung_Opacity-776.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-776.png,Lung_Opacity/masks/Lung_Opacity-776.png,True,False,299,299,L,135.53,d126b50aee17d7da5fd6b1ac2470d5c7
Lung_Opacity-777.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-777.png,Lung_Opacity/images/Lung_Opacity-777.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-777.png,Lung_Opacity/masks/Lung_Opacity-777.png,True,False,299,299,L,113.32,872c105ec564161ead3a8949bc2b6472
Lung_Opacity-778.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-778.png,Lung_Opacity/images/Lung_Opacity-778.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-778.png,Lung_Opacity/masks/Lung_Opacity-778.png,True,False,299,299,L,161.81,1c5881fb430f559517b83195d28a10cb
Lung_Opacity-779.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-779.png,Lung_Opacity/images/Lung_Opacity-779.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-779.png,Lung_Opacity/masks/Lung_Opacity-779.png,True,False,299,299,L,116.45,b8e419c2452a5aec39b051f5d39113a7
Lung_Opacity-78.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-78.png,Lung_Opacity/images/Lung_Opacity-78.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-78.png,Lung_Opacity/masks/Lung_Opacity-78.png,True,False,299,299,L,140.35,591efe37a25134db46dd75478568ebf9
Lung_Opacity-780.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-780.png,Lung_Opacity/images/Lung_Opacity-780.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-780.png,Lung_Opacity/masks/Lung_Opacity-780.png,True,False,299,299,L,150.74,cd36ae416f07a3657959896fcb42ccfc
Lung_Opacity-781.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-781.png,Lung_Opacity/images/Lung_Opacity-781.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-781.png,Lung_Opacity/masks/Lung_Opacity-781.png,True,False,299,299,L,150.35,8b31a032d8ad0accfc969a848ed6e1c8
Lung_Opacity-782.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-782.png,Lung_Opacity/images/Lung_Opacity-782.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-782.png,Lung_Opacity/masks/Lung_Opacity-782.png,True,False,299,299,L,115.9,ddd1d735d6c167008bfda130807823d1
Lung_Opacity-783.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-783.png,Lung_Opacity/images/Lung_Opacity-783.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-783.png,Lung_Opacity/masks/Lung_Opacity-783.png,True,False,299,299,L,166.16,1347ff108f0855375358d300e45ed8b1
Lung_Opacity-784.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-784.png,Lung_Opacity/images/Lung_Opacity-784.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-784.png,Lung_Opacity/masks/Lung_Opacity-784.png,True,False,299,299,L,122.86,938e335c976bd4c3282dc6663fc5e021
Lung_Opacity-785.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-785.png,Lung_Opacity/images/Lung_Opacity-785.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-785.png,Lung_Opacity/masks/Lung_Opacity-785.png,True,False,299,299,L,129.96,5c17c1b89cd18b7f94175b7b6397395a
Lung_Opacity-786.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-786.png,Lung_Opacity/images/Lung_Opacity-786.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-786.png,Lung_Opacity/masks/Lung_Opacity-786.png,True,False,299,299,L,143.13,cca5c9658922eb8791ac4e0f2f01fd2b
Lung_Opacity-787.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-787.png,Lung_Opacity/images/Lung_Opacity-787.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-787.png,Lung_Opacity/masks/Lung_Opacity-787.png,True,False,299,299,L,94.33,9650f28de1b1bbd0f75aaa022299fc12
Lung_Opacity-788.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-788.png,Lung_Opacity/images/Lung_Opacity-788.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-788.png,Lung_Opacity/masks/Lung_Opacity-788.png,True,False,299,299,L,151.54,a7a3a51e4c5b05b1466c057f1e477a92
Lung_Opacity-789.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-789.png,Lung_Opacity/images/Lung_Opacity-789.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-789.png,Lung_Opacity/masks/Lung_Opacity-789.png,True,False,299,299,L,149.84,b13ebe6bda0ce87796882e1f948564c9
Lung_Opacity-79.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-79.png,Lung_Opacity/images/Lung_Opacity-79.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-79.png,Lung_Opacity/masks/Lung_Opacity-79.png,True,False,299,299,L,123.39,b3399a22166b32d2ee94163a4a82950f
Lung_Opacity-790.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-790.png,Lung_Opacity/images/Lung_Opacity-790.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-790.png,Lung_Opacity/masks/Lung_Opacity-790.png,True,False,299,299,L,160.65,998149658d457df71bfe5c83399cf045
Lung_Opacity-791.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-791.png,Lung_Opacity/images/Lung_Opacity-791.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-791.png,Lung_Opacity/masks/Lung_Opacity-791.png,True,False,299,299,L,95.34,40bef8c607177f158f26c8644e7dcdb0
Lung_Opacity-792.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-792.png,Lung_Opacity/images/Lung_Opacity-792.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-792.png,Lung_Opacity/masks/Lung_Opacity-792.png,True,False,299,299,L,94.82,1ef335bcf39e7af528d19b012ffa593a
Lung_Opacity-793.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-793.png,Lung_Opacity/images/Lung_Opacity-793.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-793.png,Lung_Opacity/masks/Lung_Opacity-793.png,True,False,299,299,L,112.25,e9a1934afb00c551858d3d089212eb95
Lung_Opacity-794.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-794.png,Lung_Opacity/images/Lung_Opacity-794.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-794.png,Lung_Opacity/masks/Lung_Opacity-794.png,True,False,299,299,L,113.42,6a4ba5e5336b8be384c59d8faf39f86b
Lung_Opacity-795.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-795.png,Lung_Opacity/images/Lung_Opacity-795.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-795.png,Lung_Opacity/masks/Lung_Opacity-795.png,True,False,299,299,L,122.3,4ece05f7807e15029ad9689af7ed9f0c
Lung_Opacity-796.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-796.png,Lung_Opacity/images/Lung_Opacity-796.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-796.png,Lung_Opacity/masks/Lung_Opacity-796.png,True,False,299,299,L,150.1,f1add9e9fcdbd8ecdfbca08608aed8d4
Lung_Opacity-797.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-797.png,Lung_Opacity/images/Lung_Opacity-797.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-797.png,Lung_Opacity/masks/Lung_Opacity-797.png,True,False,299,299,L,103.69,26c3db8ee1a9c72907ce8da432f56e01
Lung_Opacity-798.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-798.png,Lung_Opacity/images/Lung_Opacity-798.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-798.png,Lung_Opacity/masks/Lung_Opacity-798.png,True,False,299,299,L,133.07,a8b5c881af643c70ac150b52bc3cb7dd
Lung_Opacity-799.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-799.png,Lung_Opacity/images/Lung_Opacity-799.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-799.png,Lung_Opacity/masks/Lung_Opacity-799.png,True,False,299,299,L,134.67,18185af020cb7b625b18b86fd3faba9b
Lung_Opacity-8.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-8.png,Lung_Opacity/images/Lung_Opacity-8.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-8.png,Lung_Opacity/masks/Lung_Opacity-8.png,True,False,299,299,L,183.71,ed1801943976b91140cfa3ddd5e6e568
Lung_Opacity-80.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-80.png,Lung_Opacity/images/Lung_Opacity-80.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-80.png,Lung_Opacity/masks/Lung_Opacity-80.png,True,False,299,299,L,168.08,101c94c2e9867abef84753576d6e5e12
Lung_Opacity-800.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-800.png,Lung_Opacity/images/Lung_Opacity-800.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-800.png,Lung_Opacity/masks/Lung_Opacity-800.png,True,False,299,299,L,112.87,751e7b649002b45bb708fdf531191dac
Lung_Opacity-801.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-801.png,Lung_Opacity/images/Lung_Opacity-801.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-801.png,Lung_Opacity/masks/Lung_Opacity-801.png,True,False,299,299,L,103.43,e2eba8c9f4c282f65d0dd5e426ee1b76
Lung_Opacity-802.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-802.png,Lung_Opacity/images/Lung_Opacity-802.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-802.png,Lung_Opacity/masks/Lung_Opacity-802.png,True,False,299,299,L,130.48,5164ce3a4989620b098d4c5b1904e27c
Lung_Opacity-803.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-803.png,Lung_Opacity/images/Lung_Opacity-803.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-803.png,Lung_Opacity/masks/Lung_Opacity-803.png,True,False,299,299,L,95.1,5ff85160a0eef8af6bf1757237e61602
Lung_Opacity-804.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-804.png,Lung_Opacity/images/Lung_Opacity-804.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-804.png,Lung_Opacity/masks/Lung_Opacity-804.png,True,False,299,299,L,168.37,3ef5d3a2a660180f3a3e1560670d2e2b
Lung_Opacity-805.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-805.png,Lung_Opacity/images/Lung_Opacity-805.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-805.png,Lung_Opacity/masks/Lung_Opacity-805.png,True,False,299,299,L,112.28,1f27efe179dc8428acfec183a7b894ae
Lung_Opacity-806.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-806.png,Lung_Opacity/images/Lung_Opacity-806.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-806.png,Lung_Opacity/masks/Lung_Opacity-806.png,True,False,299,299,L,159.0,632cefbacd0f147c33edbf659531cc57
Lung_Opacity-807.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-807.png,Lung_Opacity/images/Lung_Opacity-807.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-807.png,Lung_Opacity/masks/Lung_Opacity-807.png,True,False,299,299,L,117.18,f9f8bd12fb2ef969c6b464bd1715948a
Lung_Opacity-808.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-808.png,Lung_Opacity/images/Lung_Opacity-808.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-808.png,Lung_Opacity/masks/Lung_Opacity-808.png,True,False,299,299,L,117.81,f4631b6fb621adf658462a910c1d7653
Lung_Opacity-809.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-809.png,Lung_Opacity/images/Lung_Opacity-809.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-809.png,Lung_Opacity/masks/Lung_Opacity-809.png,True,False,299,299,L,99.07,6d153b1e1797076ed119c076848063a0
Lung_Opacity-81.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-81.png,Lung_Opacity/images/Lung_Opacity-81.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-81.png,Lung_Opacity/masks/Lung_Opacity-81.png,True,False,299,299,L,112.22,da129daadbb1aa06188c38c3b199056d
Lung_Opacity-810.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-810.png,Lung_Opacity/images/Lung_Opacity-810.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-810.png,Lung_Opacity/masks/Lung_Opacity-810.png,True,False,299,299,L,143.72,1cbad3378b6bbc10b20e81d8578ccc94
Lung_Opacity-811.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-811.png,Lung_Opacity/images/Lung_Opacity-811.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-811.png,Lung_Opacity/masks/Lung_Opacity-811.png,True,False,299,299,L,109.06,e457653975354982d72778706e987af1
Lung_Opacity-812.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-812.png,Lung_Opacity/images/Lung_Opacity-812.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-812.png,Lung_Opacity/masks/Lung_Opacity-812.png,True,False,299,299,L,71.22,233dc864e328172af9c7841aa59774b2
Lung_Opacity-813.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-813.png,Lung_Opacity/images/Lung_Opacity-813.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-813.png,Lung_Opacity/masks/Lung_Opacity-813.png,True,False,299,299,L,120.6,27de908ae575256c389acbb444b78bcd
Lung_Opacity-814.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-814.png,Lung_Opacity/images/Lung_Opacity-814.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-814.png,Lung_Opacity/masks/Lung_Opacity-814.png,True,False,299,299,L,164.94,37e4d9345875e89ed894f679fe6b2cf5
Lung_Opacity-815.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-815.png,Lung_Opacity/images/Lung_Opacity-815.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-815.png,Lung_Opacity/masks/Lung_Opacity-815.png,True,False,299,299,L,110.96,f9c0cd5b90644238e6d562847d3115ea
Lung_Opacity-816.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-816.png,Lung_Opacity/images/Lung_Opacity-816.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-816.png,Lung_Opacity/masks/Lung_Opacity-816.png,True,False,299,299,L,182.96,3e214193ed039ab89f1954c17cd4899c
Lung_Opacity-817.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-817.png,Lung_Opacity/images/Lung_Opacity-817.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-817.png,Lung_Opacity/masks/Lung_Opacity-817.png,True,False,299,299,L,98.73,05b1c8fc0b4ba4f3f4f37a8d52243fd3
Lung_Opacity-818.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-818.png,Lung_Opacity/images/Lung_Opacity-818.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-818.png,Lung_Opacity/masks/Lung_Opacity-818.png,True,False,299,299,L,114.7,a5e94f079f35d41efa44401bfe5e0942
Lung_Opacity-819.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-819.png,Lung_Opacity/images/Lung_Opacity-819.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-819.png,Lung_Opacity/masks/Lung_Opacity-819.png,True,False,299,299,L,164.13,836e296b3ce79c347ab3ad19d362596b
Lung_Opacity-82.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-82.png,Lung_Opacity/images/Lung_Opacity-82.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-82.png,Lung_Opacity/masks/Lung_Opacity-82.png,True,False,299,299,L,118.54,6752b0ac68b97b12453465a03538c774
Lung_Opacity-820.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-820.png,Lung_Opacity/images/Lung_Opacity-820.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-820.png,Lung_Opacity/masks/Lung_Opacity-820.png,True,False,299,299,L,105.41,05c76b522f02acabd1a472d3225ab121
Lung_Opacity-821.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-821.png,Lung_Opacity/images/Lung_Opacity-821.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-821.png,Lung_Opacity/masks/Lung_Opacity-821.png,True,False,299,299,L,152.32,ada1123f7faf149c82b35fb100a3cee6
Lung_Opacity-822.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-822.png,Lung_Opacity/images/Lung_Opacity-822.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-822.png,Lung_Opacity/masks/Lung_Opacity-822.png,True,False,299,299,L,113.19,bf6e857753da707a7b85316a5c7f8c8f
Lung_Opacity-823.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-823.png,Lung_Opacity/images/Lung_Opacity-823.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-823.png,Lung_Opacity/masks/Lung_Opacity-823.png,True,False,299,299,L,118.84,ed6f59a0524195d848aad8dfdc4f335b
Lung_Opacity-824.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-824.png,Lung_Opacity/images/Lung_Opacity-824.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-824.png,Lung_Opacity/masks/Lung_Opacity-824.png,True,False,299,299,L,118.66,0ac64b4c674be2536627cd82ce7c8d9e
Lung_Opacity-825.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-825.png,Lung_Opacity/images/Lung_Opacity-825.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-825.png,Lung_Opacity/masks/Lung_Opacity-825.png,True,False,299,299,L,129.7,0cf0a3babc321561a90619942e7a1c04
Lung_Opacity-826.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-826.png,Lung_Opacity/images/Lung_Opacity-826.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-826.png,Lung_Opacity/masks/Lung_Opacity-826.png,True,False,299,299,L,109.22,5f76178ba04df5bf456e0bb4ea5407c9
Lung_Opacity-827.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-827.png,Lung_Opacity/images/Lung_Opacity-827.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-827.png,Lung_Opacity/masks/Lung_Opacity-827.png,True,False,299,299,L,98.83,debed6596d9ddbee4c0229ba7b789d39
Lung_Opacity-828.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-828.png,Lung_Opacity/images/Lung_Opacity-828.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-828.png,Lung_Opacity/masks/Lung_Opacity-828.png,True,False,299,299,L,46.45,2cb3aad81fc6357a37b28686fa55bfc4
Lung_Opacity-829.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-829.png,Lung_Opacity/images/Lung_Opacity-829.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-829.png,Lung_Opacity/masks/Lung_Opacity-829.png,True,False,299,299,L,162.73,9458caf58361e93fde93cbb2822e01d3
Lung_Opacity-83.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-83.png,Lung_Opacity/images/Lung_Opacity-83.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-83.png,Lung_Opacity/masks/Lung_Opacity-83.png,True,False,299,299,L,131.05,f4929643e09a16d5c31b9d4520ea446e
Lung_Opacity-830.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-830.png,Lung_Opacity/images/Lung_Opacity-830.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-830.png,Lung_Opacity/masks/Lung_Opacity-830.png,True,False,299,299,L,155.57,9381adc814e37d4448f56e0b27943480
Lung_Opacity-831.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-831.png,Lung_Opacity/images/Lung_Opacity-831.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-831.png,Lung_Opacity/masks/Lung_Opacity-831.png,True,False,299,299,L,81.38,4fb6afe400cc4eef0245104803e965e6
Lung_Opacity-832.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-832.png,Lung_Opacity/images/Lung_Opacity-832.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-832.png,Lung_Opacity/masks/Lung_Opacity-832.png,True,False,299,299,L,159.13,9b9f9a4edd4778751f58fa95f1785725
Lung_Opacity-833.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-833.png,Lung_Opacity/images/Lung_Opacity-833.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-833.png,Lung_Opacity/masks/Lung_Opacity-833.png,True,False,299,299,L,167.05,e1b28637bf489eebbe4da7466acbd4ac
Lung_Opacity-834.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-834.png,Lung_Opacity/images/Lung_Opacity-834.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-834.png,Lung_Opacity/masks/Lung_Opacity-834.png,True,False,299,299,L,156.19,f4d921c119c6da043888856d7b6fc80f
Lung_Opacity-835.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-835.png,Lung_Opacity/images/Lung_Opacity-835.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-835.png,Lung_Opacity/masks/Lung_Opacity-835.png,True,False,299,299,L,166.89,7b57029727ec955b07fe641df809a74e
Lung_Opacity-836.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-836.png,Lung_Opacity/images/Lung_Opacity-836.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-836.png,Lung_Opacity/masks/Lung_Opacity-836.png,True,False,299,299,L,99.95,86d6c449e17ec5934441fd3baf20ace8
Lung_Opacity-837.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-837.png,Lung_Opacity/images/Lung_Opacity-837.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-837.png,Lung_Opacity/masks/Lung_Opacity-837.png,True,False,299,299,L,103.3,e1d577f8c0b99c329c8679545b9bdd71
Lung_Opacity-838.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-838.png,Lung_Opacity/images/Lung_Opacity-838.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-838.png,Lung_Opacity/masks/Lung_Opacity-838.png,True,False,299,299,L,113.2,9bbe29773707f9851e1b4fe924f50e72
Lung_Opacity-839.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-839.png,Lung_Opacity/images/Lung_Opacity-839.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-839.png,Lung_Opacity/masks/Lung_Opacity-839.png,True,False,299,299,L,109.31,7b89006b82de3210a74876e00366af79
Lung_Opacity-84.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-84.png,Lung_Opacity/images/Lung_Opacity-84.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-84.png,Lung_Opacity/masks/Lung_Opacity-84.png,True,False,299,299,L,103.39,36bc301383c0e7904430139e91070e23
Lung_Opacity-840.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-840.png,Lung_Opacity/images/Lung_Opacity-840.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-840.png,Lung_Opacity/masks/Lung_Opacity-840.png,True,False,299,299,L,132.15,8a21d82ac925ddc4c48790e12fc73940
Lung_Opacity-841.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-841.png,Lung_Opacity/images/Lung_Opacity-841.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-841.png,Lung_Opacity/masks/Lung_Opacity-841.png,True,False,299,299,L,133.08,40c240e9dc0c0692a32ffc73525812eb
Lung_Opacity-842.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-842.png,Lung_Opacity/images/Lung_Opacity-842.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-842.png,Lung_Opacity/masks/Lung_Opacity-842.png,True,False,299,299,L,113.3,eb4f45cc89551ee49af14a8bfbcba068
Lung_Opacity-843.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-843.png,Lung_Opacity/images/Lung_Opacity-843.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-843.png,Lung_Opacity/masks/Lung_Opacity-843.png,True,False,299,299,L,112.05,9424a4ea25fa67eb46d0a3a9065f7909
Lung_Opacity-844.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-844.png,Lung_Opacity/images/Lung_Opacity-844.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-844.png,Lung_Opacity/masks/Lung_Opacity-844.png,True,False,299,299,L,134.52,e6c1594dc4cebe50b895100a7f26210b
Lung_Opacity-845.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-845.png,Lung_Opacity/images/Lung_Opacity-845.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-845.png,Lung_Opacity/masks/Lung_Opacity-845.png,True,False,299,299,L,116.94,789a834c53c022d2a04c03537dcb9acb
Lung_Opacity-846.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-846.png,Lung_Opacity/images/Lung_Opacity-846.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-846.png,Lung_Opacity/masks/Lung_Opacity-846.png,True,False,299,299,L,127.56,600966008df6f7dba7fad846008064cd
Lung_Opacity-847.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-847.png,Lung_Opacity/images/Lung_Opacity-847.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-847.png,Lung_Opacity/masks/Lung_Opacity-847.png,True,False,299,299,L,111.34,aa10fd961e8fdc51e1fa7a964ef42fc8
Lung_Opacity-848.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-848.png,Lung_Opacity/images/Lung_Opacity-848.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-848.png,Lung_Opacity/masks/Lung_Opacity-848.png,True,False,299,299,L,125.52,39cce560c2fb0035b14486c6ab1c772c
Lung_Opacity-849.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-849.png,Lung_Opacity/images/Lung_Opacity-849.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-849.png,Lung_Opacity/masks/Lung_Opacity-849.png,True,False,299,299,L,126.02,8ffb8c8ad35172407a079b61483a5f84
Lung_Opacity-85.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-85.png,Lung_Opacity/images/Lung_Opacity-85.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-85.png,Lung_Opacity/masks/Lung_Opacity-85.png,True,False,299,299,L,110.11,4e1146445ae9704c6097e80cc4bee64b
Lung_Opacity-850.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-850.png,Lung_Opacity/images/Lung_Opacity-850.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-850.png,Lung_Opacity/masks/Lung_Opacity-850.png,True,False,299,299,L,96.92,830e6443634d2a6fb37cfcd4a72fb9a4
Lung_Opacity-851.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-851.png,Lung_Opacity/images/Lung_Opacity-851.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-851.png,Lung_Opacity/masks/Lung_Opacity-851.png,True,False,299,299,L,136.87,e4cad295bb51bc6a7fa35f77058808a0
Lung_Opacity-852.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-852.png,Lung_Opacity/images/Lung_Opacity-852.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-852.png,Lung_Opacity/masks/Lung_Opacity-852.png,True,False,299,299,L,177.39,5a9b57ebe6ef4dda335f1f9cc90f8a6d
Lung_Opacity-853.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-853.png,Lung_Opacity/images/Lung_Opacity-853.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-853.png,Lung_Opacity/masks/Lung_Opacity-853.png,True,False,299,299,L,158.63,fcb2da50f00487708349b7fc815899f4
Lung_Opacity-854.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-854.png,Lung_Opacity/images/Lung_Opacity-854.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-854.png,Lung_Opacity/masks/Lung_Opacity-854.png,True,False,299,299,L,121.76,6acc3b6fd356dc9b2ba098d1e2335a99
Lung_Opacity-855.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-855.png,Lung_Opacity/images/Lung_Opacity-855.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-855.png,Lung_Opacity/masks/Lung_Opacity-855.png,True,False,299,299,L,48.93,e0c798156a524ad7e195abf4149a2e5c
Lung_Opacity-856.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-856.png,Lung_Opacity/images/Lung_Opacity-856.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-856.png,Lung_Opacity/masks/Lung_Opacity-856.png,True,False,299,299,L,109.01,44d0859b92d57c1e3e48e711b58522da
Lung_Opacity-857.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-857.png,Lung_Opacity/images/Lung_Opacity-857.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-857.png,Lung_Opacity/masks/Lung_Opacity-857.png,True,False,299,299,L,76.98,00154d68149a2cb081c1356ca9532056
Lung_Opacity-858.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-858.png,Lung_Opacity/images/Lung_Opacity-858.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-858.png,Lung_Opacity/masks/Lung_Opacity-858.png,True,False,299,299,L,140.2,09d59c4b8176dd5f3c43f6ea2656cab6
Lung_Opacity-859.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-859.png,Lung_Opacity/images/Lung_Opacity-859.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-859.png,Lung_Opacity/masks/Lung_Opacity-859.png,True,False,299,299,L,115.65,ffd6a8e62e1f172d0dea3c0d98236c5c
Lung_Opacity-86.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-86.png,Lung_Opacity/images/Lung_Opacity-86.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-86.png,Lung_Opacity/masks/Lung_Opacity-86.png,True,False,299,299,L,186.78,526926153004949fc65cc31e4a3cdd91
Lung_Opacity-860.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-860.png,Lung_Opacity/images/Lung_Opacity-860.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-860.png,Lung_Opacity/masks/Lung_Opacity-860.png,True,False,299,299,L,106.88,ee44d8acf39c87a3dc418c296c088b63
Lung_Opacity-861.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-861.png,Lung_Opacity/images/Lung_Opacity-861.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-861.png,Lung_Opacity/masks/Lung_Opacity-861.png,True,False,299,299,L,117.36,c789a52fbd621afe73df60ea537e4cdc
Lung_Opacity-862.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-862.png,Lung_Opacity/images/Lung_Opacity-862.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-862.png,Lung_Opacity/masks/Lung_Opacity-862.png,True,False,299,299,L,111.66,fa1af80b707b6fed6edd13c6d5cf419d
Lung_Opacity-863.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-863.png,Lung_Opacity/images/Lung_Opacity-863.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-863.png,Lung_Opacity/masks/Lung_Opacity-863.png,True,False,299,299,L,117.45,b0651df3c17e6e8b8ef1988a5e13d0c8
Lung_Opacity-864.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-864.png,Lung_Opacity/images/Lung_Opacity-864.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-864.png,Lung_Opacity/masks/Lung_Opacity-864.png,True,False,299,299,L,89.74,0da753875c873307a134be24706ba7b4
Lung_Opacity-865.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-865.png,Lung_Opacity/images/Lung_Opacity-865.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-865.png,Lung_Opacity/masks/Lung_Opacity-865.png,True,False,299,299,L,118.67,7c610a7879f4e9a2b4e456525e52de4d
Lung_Opacity-866.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-866.png,Lung_Opacity/images/Lung_Opacity-866.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-866.png,Lung_Opacity/masks/Lung_Opacity-866.png,True,False,299,299,L,110.5,b8a85f1a9d6b9f227d60954f32bc3f89
Lung_Opacity-867.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-867.png,Lung_Opacity/images/Lung_Opacity-867.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-867.png,Lung_Opacity/masks/Lung_Opacity-867.png,True,False,299,299,L,144.16,53e710096f20fd6bf358eb77d010ad43
Lung_Opacity-868.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-868.png,Lung_Opacity/images/Lung_Opacity-868.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-868.png,Lung_Opacity/masks/Lung_Opacity-868.png,True,False,299,299,L,125.94,7f2ab6a669b8e5b776521da402d89759
Lung_Opacity-869.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-869.png,Lung_Opacity/images/Lung_Opacity-869.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-869.png,Lung_Opacity/masks/Lung_Opacity-869.png,True,False,299,299,L,135.83,60aac29a21dd1e9e6beec6cc632d4b5e
Lung_Opacity-87.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-87.png,Lung_Opacity/images/Lung_Opacity-87.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-87.png,Lung_Opacity/masks/Lung_Opacity-87.png,True,False,299,299,L,106.93,5dec6fc58da07687f471effe626877ff
Lung_Opacity-870.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-870.png,Lung_Opacity/images/Lung_Opacity-870.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-870.png,Lung_Opacity/masks/Lung_Opacity-870.png,True,False,299,299,L,104.51,742234e8c96466d05abd1aae4865a7fd
Lung_Opacity-871.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-871.png,Lung_Opacity/images/Lung_Opacity-871.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-871.png,Lung_Opacity/masks/Lung_Opacity-871.png,True,False,299,299,L,172.17,4310e13ee0732ab2f1b2779075ad4bbc
Lung_Opacity-872.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-872.png,Lung_Opacity/images/Lung_Opacity-872.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-872.png,Lung_Opacity/masks/Lung_Opacity-872.png,True,False,299,299,L,151.02,ed6da70e01f19fde4342160aa393ef7f
Lung_Opacity-873.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-873.png,Lung_Opacity/images/Lung_Opacity-873.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-873.png,Lung_Opacity/masks/Lung_Opacity-873.png,True,False,299,299,L,47.72,bfae9672c803fb1b6d2d8bf7bbfebe8b
Lung_Opacity-874.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-874.png,Lung_Opacity/images/Lung_Opacity-874.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-874.png,Lung_Opacity/masks/Lung_Opacity-874.png,True,False,299,299,L,139.44,94d8c6035a3d6b9a8b724d454f5941f0
Lung_Opacity-875.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-875.png,Lung_Opacity/images/Lung_Opacity-875.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-875.png,Lung_Opacity/masks/Lung_Opacity-875.png,True,False,299,299,L,105.82,b7bd201825463c892fa775d9be677931
Lung_Opacity-876.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-876.png,Lung_Opacity/images/Lung_Opacity-876.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-876.png,Lung_Opacity/masks/Lung_Opacity-876.png,True,False,299,299,L,168.98,bc4fa9ab3f3d88154df57c1b5c7a9e5f
Lung_Opacity-877.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-877.png,Lung_Opacity/images/Lung_Opacity-877.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-877.png,Lung_Opacity/masks/Lung_Opacity-877.png,True,False,299,299,L,104.47,3aca95473b828dad053dd8c1af8eed37
Lung_Opacity-878.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-878.png,Lung_Opacity/images/Lung_Opacity-878.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-878.png,Lung_Opacity/masks/Lung_Opacity-878.png,True,False,299,299,L,113.8,e30cdc7a86c2c25d4c4867f0cb9801a9
Lung_Opacity-879.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-879.png,Lung_Opacity/images/Lung_Opacity-879.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-879.png,Lung_Opacity/masks/Lung_Opacity-879.png,True,False,299,299,L,85.06,3e6231a95ef1fca53595c094d4bbcd2b
Lung_Opacity-88.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-88.png,Lung_Opacity/images/Lung_Opacity-88.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-88.png,Lung_Opacity/masks/Lung_Opacity-88.png,True,False,299,299,L,123.45,16de1974f16286a8d73a8eaadff068f6
Lung_Opacity-880.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-880.png,Lung_Opacity/images/Lung_Opacity-880.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-880.png,Lung_Opacity/masks/Lung_Opacity-880.png,True,False,299,299,L,167.15,d3f3495528afc27f313bc03ade970f98
Lung_Opacity-881.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-881.png,Lung_Opacity/images/Lung_Opacity-881.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-881.png,Lung_Opacity/masks/Lung_Opacity-881.png,True,False,299,299,L,166.96,145b8735243a703c13fc63e25a201b16
Lung_Opacity-882.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-882.png,Lung_Opacity/images/Lung_Opacity-882.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-882.png,Lung_Opacity/masks/Lung_Opacity-882.png,True,False,299,299,L,110.26,6f268158adf6108f96b757f4f94ed68d
Lung_Opacity-883.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-883.png,Lung_Opacity/images/Lung_Opacity-883.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-883.png,Lung_Opacity/masks/Lung_Opacity-883.png,True,False,299,299,L,108.31,eea9cf5348a5428ad890c090b321851a
Lung_Opacity-884.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-884.png,Lung_Opacity/images/Lung_Opacity-884.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-884.png,Lung_Opacity/masks/Lung_Opacity-884.png,True,False,299,299,L,106.7,5dc97fef9391d80725499ae54c2ac6a3
Lung_Opacity-885.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-885.png,Lung_Opacity/images/Lung_Opacity-885.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-885.png,Lung_Opacity/masks/Lung_Opacity-885.png,True,False,299,299,L,111.26,6ffe31b2fecd744a5414f0f1e0862238
Lung_Opacity-886.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-886.png,Lung_Opacity/images/Lung_Opacity-886.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-886.png,Lung_Opacity/masks/Lung_Opacity-886.png,True,False,299,299,L,116.09,3a2a0a5aa5059e9b526be8ff31d77fbb
Lung_Opacity-887.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-887.png,Lung_Opacity/images/Lung_Opacity-887.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-887.png,Lung_Opacity/masks/Lung_Opacity-887.png,True,False,299,299,L,114.52,9222d642cfb6dddbef53085087f993a5
Lung_Opacity-888.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-888.png,Lung_Opacity/images/Lung_Opacity-888.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-888.png,Lung_Opacity/masks/Lung_Opacity-888.png,True,False,299,299,L,137.52,b92d0dd98cd7621b3e22142686019552
Lung_Opacity-889.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-889.png,Lung_Opacity/images/Lung_Opacity-889.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-889.png,Lung_Opacity/masks/Lung_Opacity-889.png,True,False,299,299,L,145.37,45627046c59603cb23f023822561f8f3
Lung_Opacity-89.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-89.png,Lung_Opacity/images/Lung_Opacity-89.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-89.png,Lung_Opacity/masks/Lung_Opacity-89.png,True,False,299,299,L,158.62,405ae6531597cda68bc57412f2beb68a
Lung_Opacity-890.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-890.png,Lung_Opacity/images/Lung_Opacity-890.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-890.png,Lung_Opacity/masks/Lung_Opacity-890.png,True,False,299,299,L,156.93,85f783c0d68e3c3804d341040889d309
Lung_Opacity-891.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-891.png,Lung_Opacity/images/Lung_Opacity-891.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-891.png,Lung_Opacity/masks/Lung_Opacity-891.png,True,False,299,299,L,112.11,e79b27de04baaeb7bc18c905215cffe9
Lung_Opacity-892.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-892.png,Lung_Opacity/images/Lung_Opacity-892.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-892.png,Lung_Opacity/masks/Lung_Opacity-892.png,True,False,299,299,L,112.53,1f48eed5de12184270a4e615a78101c2
Lung_Opacity-893.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-893.png,Lung_Opacity/images/Lung_Opacity-893.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-893.png,Lung_Opacity/masks/Lung_Opacity-893.png,True,False,299,299,L,138.41,cef75a233e5121a9fd224d6022c7609b
Lung_Opacity-894.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-894.png,Lung_Opacity/images/Lung_Opacity-894.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-894.png,Lung_Opacity/masks/Lung_Opacity-894.png,True,False,299,299,L,114.14,09a6d8996069f9b3baa6a3ceea12538e
Lung_Opacity-895.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-895.png,Lung_Opacity/images/Lung_Opacity-895.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-895.png,Lung_Opacity/masks/Lung_Opacity-895.png,True,False,299,299,L,135.81,d03c63cd2c244666369f942cf1af47c3
Lung_Opacity-896.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-896.png,Lung_Opacity/images/Lung_Opacity-896.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-896.png,Lung_Opacity/masks/Lung_Opacity-896.png,True,False,299,299,L,135.07,ce4dde41bc40ed7a56fe3114faddc4ae
Lung_Opacity-897.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-897.png,Lung_Opacity/images/Lung_Opacity-897.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-897.png,Lung_Opacity/masks/Lung_Opacity-897.png,True,False,299,299,L,126.92,f7239d26f5d5e2f41c79e127431d8a23
Lung_Opacity-898.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-898.png,Lung_Opacity/images/Lung_Opacity-898.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-898.png,Lung_Opacity/masks/Lung_Opacity-898.png,True,False,299,299,L,117.93,9e8955e4517ce3dffaa576ad47fa44df
Lung_Opacity-899.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-899.png,Lung_Opacity/images/Lung_Opacity-899.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-899.png,Lung_Opacity/masks/Lung_Opacity-899.png,True,False,299,299,L,123.94,428a381b70fab721cbff718cc3e9b6ef
Lung_Opacity-9.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-9.png,Lung_Opacity/images/Lung_Opacity-9.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-9.png,Lung_Opacity/masks/Lung_Opacity-9.png,True,False,299,299,L,165.37,b536a8be6a1648f00c0cd3f4305213b1
Lung_Opacity-90.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-90.png,Lung_Opacity/images/Lung_Opacity-90.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-90.png,Lung_Opacity/masks/Lung_Opacity-90.png,True,False,299,299,L,110.38,63245bc0e041f37f90f98e314bea18f9
Lung_Opacity-900.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-900.png,Lung_Opacity/images/Lung_Opacity-900.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-900.png,Lung_Opacity/masks/Lung_Opacity-900.png,True,False,299,299,L,110.12,e5d4a7dc573175950cb2415f58424d5c
Lung_Opacity-901.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-901.png,Lung_Opacity/images/Lung_Opacity-901.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-901.png,Lung_Opacity/masks/Lung_Opacity-901.png,True,False,299,299,L,137.08,21ef66f71e1de69b95d4ce9b1515277f
Lung_Opacity-902.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-902.png,Lung_Opacity/images/Lung_Opacity-902.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-902.png,Lung_Opacity/masks/Lung_Opacity-902.png,True,False,299,299,L,128.4,c57b19586d9540ac7e6125b2ab094a9e
Lung_Opacity-903.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-903.png,Lung_Opacity/images/Lung_Opacity-903.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-903.png,Lung_Opacity/masks/Lung_Opacity-903.png,True,False,299,299,L,92.2,a1917e6ddbcdeaecc0bc020dc11db2e3
Lung_Opacity-904.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-904.png,Lung_Opacity/images/Lung_Opacity-904.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-904.png,Lung_Opacity/masks/Lung_Opacity-904.png,True,False,299,299,L,162.68,c806d26643afd88b2ce346abe39e21d0
Lung_Opacity-905.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-905.png,Lung_Opacity/images/Lung_Opacity-905.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-905.png,Lung_Opacity/masks/Lung_Opacity-905.png,True,False,299,299,L,104.22,4d37d9826f2134be73dc59138c3d4c6b
Lung_Opacity-906.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-906.png,Lung_Opacity/images/Lung_Opacity-906.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-906.png,Lung_Opacity/masks/Lung_Opacity-906.png,True,False,299,299,L,140.56,1855f6f7996f3999bd263b559c989519
Lung_Opacity-907.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-907.png,Lung_Opacity/images/Lung_Opacity-907.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-907.png,Lung_Opacity/masks/Lung_Opacity-907.png,True,False,299,299,L,115.82,adcd7e7846cfbcefa604ccfd99190a05
Lung_Opacity-908.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-908.png,Lung_Opacity/images/Lung_Opacity-908.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-908.png,Lung_Opacity/masks/Lung_Opacity-908.png,True,False,299,299,L,152.85,2e91695b86a2ae1bb75e65c0a962e6af
Lung_Opacity-909.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-909.png,Lung_Opacity/images/Lung_Opacity-909.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-909.png,Lung_Opacity/masks/Lung_Opacity-909.png,True,False,299,299,L,106.5,16a479dae1d766bde7b382780ee64647
Lung_Opacity-91.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-91.png,Lung_Opacity/images/Lung_Opacity-91.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-91.png,Lung_Opacity/masks/Lung_Opacity-91.png,True,False,299,299,L,140.76,e6536fd5d4990d2a4003e0f8a97c41c2
Lung_Opacity-910.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-910.png,Lung_Opacity/images/Lung_Opacity-910.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-910.png,Lung_Opacity/masks/Lung_Opacity-910.png,True,False,299,299,L,110.88,f8e1f1a821d65634d63c1a634d2003a5
Lung_Opacity-911.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-911.png,Lung_Opacity/images/Lung_Opacity-911.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-911.png,Lung_Opacity/masks/Lung_Opacity-911.png,True,False,299,299,L,107.28,c0b95bcbc6c8642440690d4285e137ef
Lung_Opacity-912.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-912.png,Lung_Opacity/images/Lung_Opacity-912.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-912.png,Lung_Opacity/masks/Lung_Opacity-912.png,True,False,299,299,L,156.58,9eee225506da3cbbe2fd17349794eda0
Lung_Opacity-913.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-913.png,Lung_Opacity/images/Lung_Opacity-913.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-913.png,Lung_Opacity/masks/Lung_Opacity-913.png,True,False,299,299,L,141.31,8c40e0e4b8d0467aa9c2c54355c888b7
Lung_Opacity-914.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-914.png,Lung_Opacity/images/Lung_Opacity-914.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-914.png,Lung_Opacity/masks/Lung_Opacity-914.png,True,False,299,299,L,113.83,68ca391ae2ecc1adfac65ad08be6e27f
Lung_Opacity-915.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-915.png,Lung_Opacity/images/Lung_Opacity-915.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-915.png,Lung_Opacity/masks/Lung_Opacity-915.png,True,False,299,299,L,128.39,efec8214f57ed75d3664cc7022859cd9
Lung_Opacity-916.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-916.png,Lung_Opacity/images/Lung_Opacity-916.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-916.png,Lung_Opacity/masks/Lung_Opacity-916.png,True,False,299,299,L,119.66,918c59ff9680e41152faa842138e5b58
Lung_Opacity-917.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-917.png,Lung_Opacity/images/Lung_Opacity-917.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-917.png,Lung_Opacity/masks/Lung_Opacity-917.png,True,False,299,299,L,115.99,6a7fa3f89d8855daf6a3c599b47b7ca5
Lung_Opacity-918.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-918.png,Lung_Opacity/images/Lung_Opacity-918.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-918.png,Lung_Opacity/masks/Lung_Opacity-918.png,True,False,299,299,L,114.05,ddad4890ff801b41321d10a4ad92a926
Lung_Opacity-919.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-919.png,Lung_Opacity/images/Lung_Opacity-919.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-919.png,Lung_Opacity/masks/Lung_Opacity-919.png,True,False,299,299,L,109.43,a8c0dbb9c783aa496e9b1f802b4bd608
Lung_Opacity-92.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-92.png,Lung_Opacity/images/Lung_Opacity-92.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-92.png,Lung_Opacity/masks/Lung_Opacity-92.png,True,False,299,299,L,105.17,9cce77808aaeeb5fe77bd3ed1cd727a4
Lung_Opacity-920.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-920.png,Lung_Opacity/images/Lung_Opacity-920.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-920.png,Lung_Opacity/masks/Lung_Opacity-920.png,True,False,299,299,L,106.7,e01e8b59f039e6767980c61e2bfd437d
Lung_Opacity-921.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-921.png,Lung_Opacity/images/Lung_Opacity-921.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-921.png,Lung_Opacity/masks/Lung_Opacity-921.png,True,False,299,299,L,85.57,d9fc3238cd15c34bf88dbc2f3a2d315b
Lung_Opacity-922.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-922.png,Lung_Opacity/images/Lung_Opacity-922.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-922.png,Lung_Opacity/masks/Lung_Opacity-922.png,True,False,299,299,L,113.12,f20b7d3e00dbab7594709502e1e82b93
Lung_Opacity-923.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-923.png,Lung_Opacity/images/Lung_Opacity-923.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-923.png,Lung_Opacity/masks/Lung_Opacity-923.png,True,False,299,299,L,136.39,6ad757e649e3ea90e4d73faf39afac70
Lung_Opacity-924.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-924.png,Lung_Opacity/images/Lung_Opacity-924.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-924.png,Lung_Opacity/masks/Lung_Opacity-924.png,True,False,299,299,L,152.99,98897752c26dbb6e281f4cca7b47627b
Lung_Opacity-925.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-925.png,Lung_Opacity/images/Lung_Opacity-925.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-925.png,Lung_Opacity/masks/Lung_Opacity-925.png,True,False,299,299,L,167.17,d02ae89b6f7953909b0a12748888f7d5
Lung_Opacity-926.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-926.png,Lung_Opacity/images/Lung_Opacity-926.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-926.png,Lung_Opacity/masks/Lung_Opacity-926.png,True,False,299,299,L,110.12,f4f8d487263dbfe6e77e8ecc11e0121f
Lung_Opacity-927.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-927.png,Lung_Opacity/images/Lung_Opacity-927.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-927.png,Lung_Opacity/masks/Lung_Opacity-927.png,True,False,299,299,L,108.1,6fa88bf5f9ac4b7e4c85f106f650a4ba
Lung_Opacity-928.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-928.png,Lung_Opacity/images/Lung_Opacity-928.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-928.png,Lung_Opacity/masks/Lung_Opacity-928.png,True,False,299,299,L,121.9,c9cf381f6dd14c33ae77f9151d4c2923
Lung_Opacity-929.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-929.png,Lung_Opacity/images/Lung_Opacity-929.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-929.png,Lung_Opacity/masks/Lung_Opacity-929.png,True,False,299,299,L,178.62,c22e90b27ee7534bad97790c4646a562
Lung_Opacity-93.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-93.png,Lung_Opacity/images/Lung_Opacity-93.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-93.png,Lung_Opacity/masks/Lung_Opacity-93.png,True,False,299,299,L,79.3,0b2eec7f13709db13d183da99a5fed15
Lung_Opacity-930.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-930.png,Lung_Opacity/images/Lung_Opacity-930.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-930.png,Lung_Opacity/masks/Lung_Opacity-930.png,True,False,299,299,L,113.8,e07d2b6d508609b137497eac8b3270af
Lung_Opacity-931.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-931.png,Lung_Opacity/images/Lung_Opacity-931.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-931.png,Lung_Opacity/masks/Lung_Opacity-931.png,True,False,299,299,L,159.5,996d07156f671b03c4b983a059dfabc1
Lung_Opacity-932.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-932.png,Lung_Opacity/images/Lung_Opacity-932.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-932.png,Lung_Opacity/masks/Lung_Opacity-932.png,True,False,299,299,L,134.09,b253dc14828ca455b6696712c5413ec8
Lung_Opacity-933.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-933.png,Lung_Opacity/images/Lung_Opacity-933.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-933.png,Lung_Opacity/masks/Lung_Opacity-933.png,True,False,299,299,L,109.14,63900bf1eac2bcc9ebd0f162daf321f0
Lung_Opacity-934.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-934.png,Lung_Opacity/images/Lung_Opacity-934.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-934.png,Lung_Opacity/masks/Lung_Opacity-934.png,True,False,299,299,L,100.4,45cf523377876fed961ea5012cf6ee59
Lung_Opacity-935.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-935.png,Lung_Opacity/images/Lung_Opacity-935.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-935.png,Lung_Opacity/masks/Lung_Opacity-935.png,True,False,299,299,L,173.28,702e1dc9b621d4e21137fc71229de2d3
Lung_Opacity-936.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-936.png,Lung_Opacity/images/Lung_Opacity-936.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-936.png,Lung_Opacity/masks/Lung_Opacity-936.png,True,False,299,299,L,129.27,2a7c69bf35b133ec3b75313fa5595dd4
Lung_Opacity-937.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-937.png,Lung_Opacity/images/Lung_Opacity-937.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-937.png,Lung_Opacity/masks/Lung_Opacity-937.png,True,False,299,299,L,114.71,b99dbf59c33269933a9473f5d0e069af
Lung_Opacity-938.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-938.png,Lung_Opacity/images/Lung_Opacity-938.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-938.png,Lung_Opacity/masks/Lung_Opacity-938.png,True,False,299,299,L,163.27,975c5492abc4318a1d64fa2e94db66a1
Lung_Opacity-939.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-939.png,Lung_Opacity/images/Lung_Opacity-939.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-939.png,Lung_Opacity/masks/Lung_Opacity-939.png,True,False,299,299,L,121.46,e376445c59bf469bf2a2e3b9ec495959
Lung_Opacity-94.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-94.png,Lung_Opacity/images/Lung_Opacity-94.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-94.png,Lung_Opacity/masks/Lung_Opacity-94.png,True,False,299,299,L,139.53,b8463e81548a7b2a1421af5fee62080b
Lung_Opacity-940.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-940.png,Lung_Opacity/images/Lung_Opacity-940.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-940.png,Lung_Opacity/masks/Lung_Opacity-940.png,True,False,299,299,L,115.79,26b1f5a3baeb0e9f7c0112323cfa01f5
Lung_Opacity-941.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-941.png,Lung_Opacity/images/Lung_Opacity-941.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-941.png,Lung_Opacity/masks/Lung_Opacity-941.png,True,False,299,299,L,119.64,cb829b51b4bbc5545c7470589620da97
Lung_Opacity-942.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-942.png,Lung_Opacity/images/Lung_Opacity-942.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-942.png,Lung_Opacity/masks/Lung_Opacity-942.png,True,False,299,299,L,129.68,4b218a2b0395562b16a277783940e432
Lung_Opacity-943.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-943.png,Lung_Opacity/images/Lung_Opacity-943.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-943.png,Lung_Opacity/masks/Lung_Opacity-943.png,True,False,299,299,L,153.95,04d0b8f50acec85fc1b576bd5e010307
Lung_Opacity-944.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-944.png,Lung_Opacity/images/Lung_Opacity-944.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-944.png,Lung_Opacity/masks/Lung_Opacity-944.png,True,False,299,299,L,131.36,12265e79c2721c73143cd4094234d0a2
Lung_Opacity-945.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-945.png,Lung_Opacity/images/Lung_Opacity-945.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-945.png,Lung_Opacity/masks/Lung_Opacity-945.png,True,False,299,299,L,105.68,8e5f36e78edd1f28cc54b44cde124bec
Lung_Opacity-946.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-946.png,Lung_Opacity/images/Lung_Opacity-946.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-946.png,Lung_Opacity/masks/Lung_Opacity-946.png,True,False,299,299,L,110.41,75ef2fe3a62fd0573c821381eeca2fdc
Lung_Opacity-947.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-947.png,Lung_Opacity/images/Lung_Opacity-947.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-947.png,Lung_Opacity/masks/Lung_Opacity-947.png,True,False,299,299,L,131.1,3e7342ad79fcb69520ffaf684f4e4f19
Lung_Opacity-948.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-948.png,Lung_Opacity/images/Lung_Opacity-948.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-948.png,Lung_Opacity/masks/Lung_Opacity-948.png,True,False,299,299,L,113.19,4a409baa32282cc59b58781a04c60515
Lung_Opacity-949.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-949.png,Lung_Opacity/images/Lung_Opacity-949.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-949.png,Lung_Opacity/masks/Lung_Opacity-949.png,True,False,299,299,L,142.41,9133a73400d6f1c307978bfda03f9d68
Lung_Opacity-95.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-95.png,Lung_Opacity/images/Lung_Opacity-95.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-95.png,Lung_Opacity/masks/Lung_Opacity-95.png,True,False,299,299,L,107.88,618ca3d31bfc6d88a7f757a296f1dfd7
Lung_Opacity-950.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-950.png,Lung_Opacity/images/Lung_Opacity-950.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-950.png,Lung_Opacity/masks/Lung_Opacity-950.png,True,False,299,299,L,104.31,04a68fdeab4e338a25cddb39cd1ad652
Lung_Opacity-951.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-951.png,Lung_Opacity/images/Lung_Opacity-951.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-951.png,Lung_Opacity/masks/Lung_Opacity-951.png,True,False,299,299,L,129.24,21aa7e66126734fdb53abfa060224ecd
Lung_Opacity-952.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-952.png,Lung_Opacity/images/Lung_Opacity-952.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-952.png,Lung_Opacity/masks/Lung_Opacity-952.png,True,False,299,299,L,162.27,e1d496c84f75de29bb5ca1f1622268da
Lung_Opacity-953.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-953.png,Lung_Opacity/images/Lung_Opacity-953.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-953.png,Lung_Opacity/masks/Lung_Opacity-953.png,True,False,299,299,L,118.34,d1dad7ee9612439403046a120d086b0c
Lung_Opacity-954.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-954.png,Lung_Opacity/images/Lung_Opacity-954.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-954.png,Lung_Opacity/masks/Lung_Opacity-954.png,True,False,299,299,L,134.71,7467eafb1ab983516172d3b1b0728e7a
Lung_Opacity-955.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-955.png,Lung_Opacity/images/Lung_Opacity-955.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-955.png,Lung_Opacity/masks/Lung_Opacity-955.png,True,False,299,299,L,120.53,c9f7203a88380c0a7786f8b6de4ffe66
Lung_Opacity-956.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-956.png,Lung_Opacity/images/Lung_Opacity-956.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-956.png,Lung_Opacity/masks/Lung_Opacity-956.png,True,False,299,299,L,105.65,fcf60d318a927f3330a12dec27a80094
Lung_Opacity-957.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-957.png,Lung_Opacity/images/Lung_Opacity-957.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-957.png,Lung_Opacity/masks/Lung_Opacity-957.png,True,False,299,299,L,92.24,6a0637f9f275c8b4da071232578fde2a
Lung_Opacity-958.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-958.png,Lung_Opacity/images/Lung_Opacity-958.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-958.png,Lung_Opacity/masks/Lung_Opacity-958.png,True,False,299,299,L,105.12,38b5ba1225a5b6625545d005bd1c717b
Lung_Opacity-959.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-959.png,Lung_Opacity/images/Lung_Opacity-959.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-959.png,Lung_Opacity/masks/Lung_Opacity-959.png,True,False,299,299,L,119.98,87b16f3f42a73c7367c46d9a09385cab
Lung_Opacity-96.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-96.png,Lung_Opacity/images/Lung_Opacity-96.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-96.png,Lung_Opacity/masks/Lung_Opacity-96.png,True,False,299,299,L,136.91,4a8e3b57f9eb519ef55b70e2e32e2ad2
Lung_Opacity-960.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-960.png,Lung_Opacity/images/Lung_Opacity-960.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-960.png,Lung_Opacity/masks/Lung_Opacity-960.png,True,False,299,299,L,121.68,394a47a2469d70ae3094a30fdbf2656a
Lung_Opacity-961.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-961.png,Lung_Opacity/images/Lung_Opacity-961.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-961.png,Lung_Opacity/masks/Lung_Opacity-961.png,True,False,299,299,L,128.48,db5ed13e738051532cef969e2d658d37
Lung_Opacity-962.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-962.png,Lung_Opacity/images/Lung_Opacity-962.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-962.png,Lung_Opacity/masks/Lung_Opacity-962.png,True,False,299,299,L,128.8,b17845403d92ada4857b7dcc1394b7ae
Lung_Opacity-963.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-963.png,Lung_Opacity/images/Lung_Opacity-963.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-963.png,Lung_Opacity/masks/Lung_Opacity-963.png,True,False,299,299,L,139.38,bfbc9bba85feb4af51257e40df15f7ac
Lung_Opacity-964.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-964.png,Lung_Opacity/images/Lung_Opacity-964.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-964.png,Lung_Opacity/masks/Lung_Opacity-964.png,True,False,299,299,L,98.59,ae211ba03e49898dcf19c3479e2336e4
Lung_Opacity-965.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-965.png,Lung_Opacity/images/Lung_Opacity-965.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-965.png,Lung_Opacity/masks/Lung_Opacity-965.png,True,False,299,299,L,117.04,1a46ca96e08eab9a0b945564d828ae31
Lung_Opacity-966.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-966.png,Lung_Opacity/images/Lung_Opacity-966.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-966.png,Lung_Opacity/masks/Lung_Opacity-966.png,True,False,299,299,L,113.96,f468cbe6ba10ca074360172ac6614816
Lung_Opacity-967.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-967.png,Lung_Opacity/images/Lung_Opacity-967.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-967.png,Lung_Opacity/masks/Lung_Opacity-967.png,True,False,299,299,L,116.31,4e5efe6be6bd0fd9192e4d99bb87101d
Lung_Opacity-968.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-968.png,Lung_Opacity/images/Lung_Opacity-968.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-968.png,Lung_Opacity/masks/Lung_Opacity-968.png,True,False,299,299,L,120.04,4aa10b68c3bf03c270e3ff7f7f647f38
Lung_Opacity-969.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-969.png,Lung_Opacity/images/Lung_Opacity-969.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-969.png,Lung_Opacity/masks/Lung_Opacity-969.png,True,False,299,299,L,145.02,e0fbea923631c94133bb3727bd9f584f
Lung_Opacity-97.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-97.png,Lung_Opacity/images/Lung_Opacity-97.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-97.png,Lung_Opacity/masks/Lung_Opacity-97.png,True,False,299,299,L,125.96,32b11f692e874401b1fb7ad41e4bcba8
Lung_Opacity-970.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-970.png,Lung_Opacity/images/Lung_Opacity-970.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-970.png,Lung_Opacity/masks/Lung_Opacity-970.png,True,False,299,299,L,102.78,e38d28916fa91590305400d2e7dd3f64
Lung_Opacity-971.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-971.png,Lung_Opacity/images/Lung_Opacity-971.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-971.png,Lung_Opacity/masks/Lung_Opacity-971.png,True,False,299,299,L,139.34,0b73b45586ff152713e420276e081d70
Lung_Opacity-972.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-972.png,Lung_Opacity/images/Lung_Opacity-972.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-972.png,Lung_Opacity/masks/Lung_Opacity-972.png,True,False,299,299,L,95.36,852150cf34011095d2b4b1a1474e336c
Lung_Opacity-973.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-973.png,Lung_Opacity/images/Lung_Opacity-973.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-973.png,Lung_Opacity/masks/Lung_Opacity-973.png,True,False,299,299,L,114.79,f8a3c3c613934b23ffcb18121b0117f6
Lung_Opacity-974.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-974.png,Lung_Opacity/images/Lung_Opacity-974.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-974.png,Lung_Opacity/masks/Lung_Opacity-974.png,True,False,299,299,L,106.41,62e3c3124f5c4d63a06ce15602f5ee5c
Lung_Opacity-975.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-975.png,Lung_Opacity/images/Lung_Opacity-975.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-975.png,Lung_Opacity/masks/Lung_Opacity-975.png,True,False,299,299,L,157.45,cb76a472325d15d596f3200765cf4785
Lung_Opacity-976.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-976.png,Lung_Opacity/images/Lung_Opacity-976.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-976.png,Lung_Opacity/masks/Lung_Opacity-976.png,True,False,299,299,L,111.85,97c3920c6ae48a4f76bab19d0b159528
Lung_Opacity-977.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-977.png,Lung_Opacity/images/Lung_Opacity-977.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-977.png,Lung_Opacity/masks/Lung_Opacity-977.png,True,False,299,299,L,122.36,7e755dacf69f1c22f953b8cb0deb16c6
Lung_Opacity-978.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-978.png,Lung_Opacity/images/Lung_Opacity-978.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-978.png,Lung_Opacity/masks/Lung_Opacity-978.png,True,False,299,299,L,109.78,8b1474018eab4eab0ed6f00b9ce53fb3
Lung_Opacity-979.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-979.png,Lung_Opacity/images/Lung_Opacity-979.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-979.png,Lung_Opacity/masks/Lung_Opacity-979.png,True,False,299,299,L,136.9,f6926552409a3f19a0bdb59ad6ba9927
Lung_Opacity-98.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-98.png,Lung_Opacity/images/Lung_Opacity-98.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-98.png,Lung_Opacity/masks/Lung_Opacity-98.png,True,False,299,299,L,160.91,be5ba5c6989294edb6486e8b886e092d
Lung_Opacity-980.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-980.png,Lung_Opacity/images/Lung_Opacity-980.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-980.png,Lung_Opacity/masks/Lung_Opacity-980.png,True,False,299,299,L,120.46,be6fb09bc55c0927ddefe3ea73c2142e
Lung_Opacity-981.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-981.png,Lung_Opacity/images/Lung_Opacity-981.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-981.png,Lung_Opacity/masks/Lung_Opacity-981.png,True,False,299,299,L,128.25,e794d225181fc15fca1e29c7277589f4
Lung_Opacity-982.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-982.png,Lung_Opacity/images/Lung_Opacity-982.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-982.png,Lung_Opacity/masks/Lung_Opacity-982.png,True,False,299,299,L,170.94,0db8a19d191b6c236d1851d7ef341e02
Lung_Opacity-983.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-983.png,Lung_Opacity/images/Lung_Opacity-983.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-983.png,Lung_Opacity/masks/Lung_Opacity-983.png,True,False,299,299,L,102.74,dc2d4d87540ccb23b919a3c97a847755
Lung_Opacity-984.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-984.png,Lung_Opacity/images/Lung_Opacity-984.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-984.png,Lung_Opacity/masks/Lung_Opacity-984.png,True,False,299,299,L,137.41,e0d36ea10bbda2d3f302adb97d57b09a
Lung_Opacity-985.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-985.png,Lung_Opacity/images/Lung_Opacity-985.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-985.png,Lung_Opacity/masks/Lung_Opacity-985.png,True,False,299,299,L,92.06,5e88e84d772a6ab0b36186041c9737fa
Lung_Opacity-986.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-986.png,Lung_Opacity/images/Lung_Opacity-986.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-986.png,Lung_Opacity/masks/Lung_Opacity-986.png,True,False,299,299,L,132.74,13c7ec7e9b892a76adae1497709b666e
Lung_Opacity-987.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-987.png,Lung_Opacity/images/Lung_Opacity-987.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-987.png,Lung_Opacity/masks/Lung_Opacity-987.png,True,False,299,299,L,135.45,46299363d643e4d07e606f91abd0b629
Lung_Opacity-988.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-988.png,Lung_Opacity/images/Lung_Opacity-988.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-988.png,Lung_Opacity/masks/Lung_Opacity-988.png,True,False,299,299,L,157.4,ccdf0d381caaecb87d59227d8d696eae
Lung_Opacity-989.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-989.png,Lung_Opacity/images/Lung_Opacity-989.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-989.png,Lung_Opacity/masks/Lung_Opacity-989.png,True,False,299,299,L,159.95,4bd767f29fd300e9317a41663845933a
Lung_Opacity-99.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-99.png,Lung_Opacity/images/Lung_Opacity-99.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-99.png,Lung_Opacity/masks/Lung_Opacity-99.png,True,False,299,299,L,55.68,b77343958641e8ddd1c68a621a71b6fa
Lung_Opacity-990.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-990.png,Lung_Opacity/images/Lung_Opacity-990.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-990.png,Lung_Opacity/masks/Lung_Opacity-990.png,True,False,299,299,L,103.89,d3bec390ef02c40e455d2ba3ef928cbb
Lung_Opacity-991.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-991.png,Lung_Opacity/images/Lung_Opacity-991.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-991.png,Lung_Opacity/masks/Lung_Opacity-991.png,True,False,299,299,L,111.77,eea07f0579c336ccd72730182d2f52eb
Lung_Opacity-992.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-992.png,Lung_Opacity/images/Lung_Opacity-992.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-992.png,Lung_Opacity/masks/Lung_Opacity-992.png,True,False,299,299,L,163.1,81f8294ed0317ce933adcd7e3fe5ce4e
Lung_Opacity-993.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-993.png,Lung_Opacity/images/Lung_Opacity-993.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-993.png,Lung_Opacity/masks/Lung_Opacity-993.png,True,False,299,299,L,134.34,55da884579906c989aa746c02f1da8ab
Lung_Opacity-994.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-994.png,Lung_Opacity/images/Lung_Opacity-994.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-994.png,Lung_Opacity/masks/Lung_Opacity-994.png,True,False,299,299,L,113.74,8bb565f3fdc32f4eb83753ff384d21ba
Lung_Opacity-995.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-995.png,Lung_Opacity/images/Lung_Opacity-995.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-995.png,Lung_Opacity/masks/Lung_Opacity-995.png,True,False,299,299,L,128.75,6d23e0e1417de5d01e04581a26244882
Lung_Opacity-996.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-996.png,Lung_Opacity/images/Lung_Opacity-996.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-996.png,Lung_Opacity/masks/Lung_Opacity-996.png,True,False,299,299,L,119.85,6aee3656d1e696ad51c2db762cdaffd8
Lung_Opacity-997.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-997.png,Lung_Opacity/images/Lung_Opacity-997.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-997.png,Lung_Opacity/masks/Lung_Opacity-997.png,True,False,299,299,L,112.64,91ef6a6600ade3c298d0dc3e1f8d485b
Lung_Opacity-998.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-998.png,Lung_Opacity/images/Lung_Opacity-998.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-998.png,Lung_Opacity/masks/Lung_Opacity-998.png,True,False,299,299,L,123.25,7cfec70e001b994eaee997494e863e51
Lung_Opacity-999.png,Lung_Opacity,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images/Lung_Opacity-999.png,Lung_Opacity/images/Lung_Opacity-999.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/masks/Lung_Opacity-999.png,Lung_Opacity/masks/Lung_Opacity-999.png,True,False,299,299,L,85.41,f9578f6b266dfda2d27654727c22eaed
Normal-1.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1.png,Normal/images/Normal-1.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1.png,Normal/masks/Normal-1.png,True,False,299,299,L,85.19,973387ba59b2669d570d73dcfe55f88e
Normal-10.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10.png,Normal/images/Normal-10.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10.png,Normal/masks/Normal-10.png,True,False,299,299,L,128.52,95aea838eb8d7ee0522e1abe0747129e
Normal-100.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-100.png,Normal/images/Normal-100.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-100.png,Normal/masks/Normal-100.png,True,False,299,299,L,125.68,829b755f1aadbec50adbd1815ffe81bf
Normal-1000.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1000.png,Normal/images/Normal-1000.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1000.png,Normal/masks/Normal-1000.png,True,False,299,299,L,113.72,69c20913b20b88e8873d188ea3fbe58d
Normal-10000.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10000.png,Normal/images/Normal-10000.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10000.png,Normal/masks/Normal-10000.png,True,False,299,299,L,131.35,b7c38ca5ff4be275d6d616545bdf0866
Normal-10001.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10001.png,Normal/images/Normal-10001.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10001.png,Normal/masks/Normal-10001.png,True,False,299,299,L,128.17,d1a54f3d0429059729196b6a7e2fd262
Normal-10002.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10002.png,Normal/images/Normal-10002.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10002.png,Normal/masks/Normal-10002.png,True,False,299,299,L,106.1,17d5983b6b0bd36c36ee80fac6c2a798
Normal-10003.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10003.png,Normal/images/Normal-10003.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10003.png,Normal/masks/Normal-10003.png,True,False,299,299,L,116.75,33b814032f6c80bd84d48413fffce1a9
Normal-10004.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10004.png,Normal/images/Normal-10004.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10004.png,Normal/masks/Normal-10004.png,True,False,299,299,L,137.0,d3ee948d6e8968c6778e4a1915a8c8db
Normal-10005.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10005.png,Normal/images/Normal-10005.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10005.png,Normal/masks/Normal-10005.png,True,False,299,299,L,174.55,59777c7f37e815a189296e2b829629de
Normal-10006.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10006.png,Normal/images/Normal-10006.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10006.png,Normal/masks/Normal-10006.png,True,False,299,299,L,118.78,409142b5e063441263029525a4906ce1
Normal-10007.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10007.png,Normal/images/Normal-10007.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10007.png,Normal/masks/Normal-10007.png,True,False,299,299,L,100.92,bcc9dcdafb05f9714254198aed1a18fd
Normal-10008.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10008.png,Normal/images/Normal-10008.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10008.png,Normal/masks/Normal-10008.png,True,False,299,299,L,121.76,653ab2012b31b120dd082cec0e03289c
Normal-10009.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10009.png,Normal/images/Normal-10009.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10009.png,Normal/masks/Normal-10009.png,True,False,299,299,L,113.76,b5bf09e4ec68d9aaac9d0b4ae04c7e85
Normal-1001.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1001.png,Normal/images/Normal-1001.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1001.png,Normal/masks/Normal-1001.png,True,False,299,299,L,95.26,c82087ec333834a44b70057660290de2
Normal-10010.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10010.png,Normal/images/Normal-10010.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10010.png,Normal/masks/Normal-10010.png,True,False,299,299,L,91.8,598b4b41602bd2701f3871d02cf4297f
Normal-10011.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10011.png,Normal/images/Normal-10011.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10011.png,Normal/masks/Normal-10011.png,True,False,299,299,L,122.64,65b3da4830d1cd02422b1489e7f45126
Normal-10012.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10012.png,Normal/images/Normal-10012.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10012.png,Normal/masks/Normal-10012.png,True,False,299,299,L,154.44,65f0e48c71ae7579a72efc56b1880785
Normal-10013.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10013.png,Normal/images/Normal-10013.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10013.png,Normal/masks/Normal-10013.png,True,False,299,299,L,140.71,1017501837e6d1b887584430cc20f930
Normal-10014.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10014.png,Normal/images/Normal-10014.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10014.png,Normal/masks/Normal-10014.png,True,False,299,299,L,188.53,8a08475dfd745070f59a50edf8f1ee60
Normal-10015.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10015.png,Normal/images/Normal-10015.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10015.png,Normal/masks/Normal-10015.png,True,False,299,299,L,115.84,f07f147de4330b357bff6147a1cfd9b1
Normal-10016.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10016.png,Normal/images/Normal-10016.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10016.png,Normal/masks/Normal-10016.png,True,False,299,299,L,134.13,7ade1d2c6be5359f3747de2e78b4bb16
Normal-10017.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10017.png,Normal/images/Normal-10017.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10017.png,Normal/masks/Normal-10017.png,True,False,299,299,L,165.26,8782f281974545346343e649dfa10f1a
Normal-10018.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10018.png,Normal/images/Normal-10018.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10018.png,Normal/masks/Normal-10018.png,True,False,299,299,L,180.89,4a1985164b4d4d3afbf56b2f96dbc996
Normal-10019.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10019.png,Normal/images/Normal-10019.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10019.png,Normal/masks/Normal-10019.png,True,False,299,299,L,100.95,d1801bf800c126f7df4077ef1a7d1ba5
Normal-1002.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1002.png,Normal/images/Normal-1002.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1002.png,Normal/masks/Normal-1002.png,True,False,299,299,L,128.29,94faeab2f1ef0866de29b3e0d9fcfd02
Normal-10020.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10020.png,Normal/images/Normal-10020.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10020.png,Normal/masks/Normal-10020.png,True,False,299,299,L,125.3,ee7e662c3d2f3fe9e3c2ea4d516a1b6d
Normal-10021.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10021.png,Normal/images/Normal-10021.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10021.png,Normal/masks/Normal-10021.png,True,False,299,299,L,89.19,aab095a51d55a12b4db3918385288dd1
Normal-10022.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10022.png,Normal/images/Normal-10022.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10022.png,Normal/masks/Normal-10022.png,True,False,299,299,L,148.45,37f55d822c764b5a05d0dad8d72d0707
Normal-10023.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10023.png,Normal/images/Normal-10023.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10023.png,Normal/masks/Normal-10023.png,True,False,299,299,L,134.05,8b9320d99309e2080b38bf3c3411e16e
Normal-10024.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10024.png,Normal/images/Normal-10024.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10024.png,Normal/masks/Normal-10024.png,True,False,299,299,L,167.31,b8781fcd48b0341db85f6d617bb71c27
Normal-10025.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10025.png,Normal/images/Normal-10025.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10025.png,Normal/masks/Normal-10025.png,True,False,299,299,L,148.71,ca415fa80208d7d35397b0a13d726ef1
Normal-10026.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10026.png,Normal/images/Normal-10026.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10026.png,Normal/masks/Normal-10026.png,True,False,299,299,L,105.24,3577d64897777ffa7c3dbfd61b2f0409
Normal-10027.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10027.png,Normal/images/Normal-10027.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10027.png,Normal/masks/Normal-10027.png,True,False,299,299,L,41.87,78c3a0d31d948e375af4a6b3f9e2d0e7
Normal-10028.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10028.png,Normal/images/Normal-10028.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10028.png,Normal/masks/Normal-10028.png,True,False,299,299,L,91.3,14621e177f095c8b0fc3f5f3999c8558
Normal-10029.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10029.png,Normal/images/Normal-10029.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10029.png,Normal/masks/Normal-10029.png,True,False,299,299,L,111.73,a0c99e8a8d4567cf433dc830f3651abf
Normal-1003.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1003.png,Normal/images/Normal-1003.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1003.png,Normal/masks/Normal-1003.png,True,False,299,299,L,116.49,6ef910faa8acaa01c6eaa54195228f7e
Normal-10030.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10030.png,Normal/images/Normal-10030.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10030.png,Normal/masks/Normal-10030.png,True,False,299,299,L,108.93,b21049e606f8293c11292c3d4b2a18d7
Normal-10031.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10031.png,Normal/images/Normal-10031.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10031.png,Normal/masks/Normal-10031.png,True,False,299,299,L,131.14,c111b3d9c8fa3e6af130a8c611b65017
Normal-10032.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10032.png,Normal/images/Normal-10032.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10032.png,Normal/masks/Normal-10032.png,True,False,299,299,L,114.71,ae2e2b2fdeab6f78fc51ce9351e3993e
Normal-10033.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10033.png,Normal/images/Normal-10033.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10033.png,Normal/masks/Normal-10033.png,True,False,299,299,L,113.98,ba5b56765b7e6d534df618dae12a6b12
Normal-10034.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10034.png,Normal/images/Normal-10034.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10034.png,Normal/masks/Normal-10034.png,True,False,299,299,L,170.17,907ea7ffec23fa332de3971a0916a8f3
Normal-10035.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10035.png,Normal/images/Normal-10035.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10035.png,Normal/masks/Normal-10035.png,True,False,299,299,L,132.14,c127356d3224fa7dad93db6883e50767
Normal-10036.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10036.png,Normal/images/Normal-10036.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10036.png,Normal/masks/Normal-10036.png,True,False,299,299,L,166.61,28cf9b1d83492e17ee4098e8cccf59e4
Normal-10037.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10037.png,Normal/images/Normal-10037.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10037.png,Normal/masks/Normal-10037.png,True,False,299,299,L,123.44,353a5b5a1e27629b106df2b6cd6cbcd3
Normal-10038.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10038.png,Normal/images/Normal-10038.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10038.png,Normal/masks/Normal-10038.png,True,False,299,299,L,171.79,34d118598c4eaa525f33e26d6574ec44
Normal-10039.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10039.png,Normal/images/Normal-10039.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10039.png,Normal/masks/Normal-10039.png,True,False,299,299,L,116.53,6dc5f779cce054f89f6d1ce5751438cc
Normal-1004.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1004.png,Normal/images/Normal-1004.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1004.png,Normal/masks/Normal-1004.png,True,False,299,299,L,123.01,e6b455e617101fd7b5a35a67c0df0a53
Normal-10040.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10040.png,Normal/images/Normal-10040.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10040.png,Normal/masks/Normal-10040.png,True,False,299,299,L,88.27,4e3ce437415f918876812a7dce15fc33
Normal-10041.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10041.png,Normal/images/Normal-10041.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10041.png,Normal/masks/Normal-10041.png,True,False,299,299,L,116.99,f13fcee789f82bec1072bb4d2ecbb92e
Normal-10042.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10042.png,Normal/images/Normal-10042.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10042.png,Normal/masks/Normal-10042.png,True,False,299,299,L,154.35,cfd738f3624de42c67f7ba6b7a3e484a
Normal-10043.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10043.png,Normal/images/Normal-10043.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10043.png,Normal/masks/Normal-10043.png,True,False,299,299,L,120.27,832c441efb0795689bcd946c20458bbc
Normal-10044.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10044.png,Normal/images/Normal-10044.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10044.png,Normal/masks/Normal-10044.png,True,False,299,299,L,155.2,db6da480dcdce1ec8b2818ad879e1787
Normal-10045.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10045.png,Normal/images/Normal-10045.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10045.png,Normal/masks/Normal-10045.png,True,False,299,299,L,127.69,7e69b0815ea05a96b60d7f517147acec
Normal-10046.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10046.png,Normal/images/Normal-10046.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10046.png,Normal/masks/Normal-10046.png,True,False,299,299,L,175.36,3239ce2dfd4a7aae9270a4ebb14f6554
Normal-10047.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10047.png,Normal/images/Normal-10047.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10047.png,Normal/masks/Normal-10047.png,True,False,299,299,L,173.53,331658c1a9d327fbd00ca52c3acc4c53
Normal-10048.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10048.png,Normal/images/Normal-10048.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10048.png,Normal/masks/Normal-10048.png,True,False,299,299,L,132.75,f6f00cde6e70a8127c390f676d0bc030
Normal-10049.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10049.png,Normal/images/Normal-10049.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10049.png,Normal/masks/Normal-10049.png,True,False,299,299,L,126.57,9b1951ed68daef083148cb5bf0737ee5
Normal-1005.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1005.png,Normal/images/Normal-1005.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1005.png,Normal/masks/Normal-1005.png,True,False,299,299,L,119.91,ab0405d5ca8ab16d01d0fcb981f9203b
Normal-10050.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10050.png,Normal/images/Normal-10050.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10050.png,Normal/masks/Normal-10050.png,True,False,299,299,L,159.81,c5151b4a9f58af6e335f833e940385d5
Normal-10051.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10051.png,Normal/images/Normal-10051.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10051.png,Normal/masks/Normal-10051.png,True,False,299,299,L,111.04,6265926ee96dece60f9fecdb1c976c02
Normal-10052.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10052.png,Normal/images/Normal-10052.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10052.png,Normal/masks/Normal-10052.png,True,False,299,299,L,108.68,9865d6601388b1d581055c632ed7fdd1
Normal-10053.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10053.png,Normal/images/Normal-10053.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10053.png,Normal/masks/Normal-10053.png,True,False,299,299,L,107.38,a7339ce9fee92df9158fd9e3ceabecc6
Normal-10054.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10054.png,Normal/images/Normal-10054.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10054.png,Normal/masks/Normal-10054.png,True,False,299,299,L,156.98,923ec834889d4554d84d208c21eedff0
Normal-10055.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10055.png,Normal/images/Normal-10055.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10055.png,Normal/masks/Normal-10055.png,True,False,299,299,L,81.91,1c75addd9ec1de1f1f9d245631a049e4
Normal-10056.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10056.png,Normal/images/Normal-10056.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10056.png,Normal/masks/Normal-10056.png,True,False,299,299,L,127.99,0bac8f0098f19533a61d51b6e816649f
Normal-10057.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10057.png,Normal/images/Normal-10057.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10057.png,Normal/masks/Normal-10057.png,True,False,299,299,L,124.0,b26ec49c499c0210cb8d35c412f66ec7
Normal-10058.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10058.png,Normal/images/Normal-10058.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10058.png,Normal/masks/Normal-10058.png,True,False,299,299,L,128.86,1a3a593d75bf330d9b9843e903d8821d
Normal-10059.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10059.png,Normal/images/Normal-10059.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10059.png,Normal/masks/Normal-10059.png,True,False,299,299,L,158.25,ae3c675958ac184084bae53b224e53d8
Normal-1006.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1006.png,Normal/images/Normal-1006.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1006.png,Normal/masks/Normal-1006.png,True,False,299,299,L,111.96,36ac577bedc02cbd639ea9a6d74e897a
Normal-10060.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10060.png,Normal/images/Normal-10060.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10060.png,Normal/masks/Normal-10060.png,True,False,299,299,L,146.52,6686c9a0037c3ea0b0ad6241f1eceb84
Normal-10061.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10061.png,Normal/images/Normal-10061.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10061.png,Normal/masks/Normal-10061.png,True,False,299,299,L,131.29,8f783d2aa61bc46525868304b3b7d1ab
Normal-10062.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10062.png,Normal/images/Normal-10062.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10062.png,Normal/masks/Normal-10062.png,True,False,299,299,L,128.16,66f097ecc56f56873fe9d4017916966e
Normal-10063.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10063.png,Normal/images/Normal-10063.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10063.png,Normal/masks/Normal-10063.png,True,False,299,299,L,120.02,58a4d4e045f5a7ae176601d8cd6dbd4f
Normal-10064.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10064.png,Normal/images/Normal-10064.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10064.png,Normal/masks/Normal-10064.png,True,False,299,299,L,131.86,955543b731759ae6fa03dfa79d7aebf8
Normal-10065.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10065.png,Normal/images/Normal-10065.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10065.png,Normal/masks/Normal-10065.png,True,False,299,299,L,112.78,27d2bf7b85125197610cf76843e23dab
Normal-10066.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10066.png,Normal/images/Normal-10066.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10066.png,Normal/masks/Normal-10066.png,True,False,299,299,L,116.52,ca2c05f724a96ebffa338bbbdf4a56e7
Normal-10067.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10067.png,Normal/images/Normal-10067.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10067.png,Normal/masks/Normal-10067.png,True,False,299,299,L,115.87,6b4c52a0531f0565a0129323881dc77e
Normal-10068.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10068.png,Normal/images/Normal-10068.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10068.png,Normal/masks/Normal-10068.png,True,False,299,299,L,138.23,8cbd318055ba5d12bd085a9bebae4de8
Normal-10069.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10069.png,Normal/images/Normal-10069.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10069.png,Normal/masks/Normal-10069.png,True,False,299,299,L,147.25,c52cc783b452163b2adc3c9bffd652f8
Normal-1007.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1007.png,Normal/images/Normal-1007.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1007.png,Normal/masks/Normal-1007.png,True,False,299,299,L,127.45,ad3acf599ce6e3784fea662b81e2ecc7
Normal-10070.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10070.png,Normal/images/Normal-10070.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10070.png,Normal/masks/Normal-10070.png,True,False,299,299,L,101.91,2c6b2e1ffa5afdf863c49201812f763b
Normal-10071.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10071.png,Normal/images/Normal-10071.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10071.png,Normal/masks/Normal-10071.png,True,False,299,299,L,111.45,d0562c72541bb95f04564b7d5398823b
Normal-10072.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10072.png,Normal/images/Normal-10072.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10072.png,Normal/masks/Normal-10072.png,True,False,299,299,L,127.48,57dd9ccc9f921f092c24c859a5d6474e
Normal-10073.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10073.png,Normal/images/Normal-10073.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10073.png,Normal/masks/Normal-10073.png,True,False,299,299,L,121.28,99d238f594d27271d94afc27bf5a3d2a
Normal-10074.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10074.png,Normal/images/Normal-10074.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10074.png,Normal/masks/Normal-10074.png,True,False,299,299,L,130.73,5bc39dd06d330d600b4ffb331e754e4f
Normal-10075.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10075.png,Normal/images/Normal-10075.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10075.png,Normal/masks/Normal-10075.png,True,False,299,299,L,180.77,0a5914092e91de699363b0dfc1dfd85e
Normal-10076.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10076.png,Normal/images/Normal-10076.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10076.png,Normal/masks/Normal-10076.png,True,False,299,299,L,118.19,162c6ddb7e2ef50d80df15e4dc180dd3
Normal-10077.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10077.png,Normal/images/Normal-10077.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10077.png,Normal/masks/Normal-10077.png,True,False,299,299,L,170.22,27ddfa163b3ab1e39c40f663397b1235
Normal-10078.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10078.png,Normal/images/Normal-10078.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10078.png,Normal/masks/Normal-10078.png,True,False,299,299,L,179.44,c586374c66374148a60f9fae700d19cd
Normal-10079.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10079.png,Normal/images/Normal-10079.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10079.png,Normal/masks/Normal-10079.png,True,False,299,299,L,131.39,3ad87cdc85a5af9462655880e6f36d56
Normal-1008.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1008.png,Normal/images/Normal-1008.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1008.png,Normal/masks/Normal-1008.png,True,False,299,299,L,156.21,44ee5db579c032f7d91e3636f978cb3f
Normal-10080.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10080.png,Normal/images/Normal-10080.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10080.png,Normal/masks/Normal-10080.png,True,False,299,299,L,138.66,fe3248d4e1fd60a300ae56cf5882d3ab
Normal-10081.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10081.png,Normal/images/Normal-10081.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10081.png,Normal/masks/Normal-10081.png,True,False,299,299,L,122.22,8e26d55ea1be715e083bc6b942b299f5
Normal-10082.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10082.png,Normal/images/Normal-10082.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10082.png,Normal/masks/Normal-10082.png,True,False,299,299,L,102.04,65cc8242bd9b8482b5ad74c4dbaa8130
Normal-10083.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10083.png,Normal/images/Normal-10083.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10083.png,Normal/masks/Normal-10083.png,True,False,299,299,L,115.74,a122789fc39c566b149238b9e11e4cc5
Normal-10084.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10084.png,Normal/images/Normal-10084.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10084.png,Normal/masks/Normal-10084.png,True,False,299,299,L,153.09,a49300d0bf9204c225168ef74595e91b
Normal-10085.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10085.png,Normal/images/Normal-10085.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10085.png,Normal/masks/Normal-10085.png,True,False,299,299,L,115.51,dec15295c3241d233e6d663b4e9ebc54
Normal-10086.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10086.png,Normal/images/Normal-10086.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10086.png,Normal/masks/Normal-10086.png,True,False,299,299,L,132.91,c21ef126eb0717e97e40b1e2e3dde054
Normal-10087.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10087.png,Normal/images/Normal-10087.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10087.png,Normal/masks/Normal-10087.png,True,False,299,299,L,145.02,117de91f0ccb87b1b24564e161b87197
Normal-10088.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10088.png,Normal/images/Normal-10088.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10088.png,Normal/masks/Normal-10088.png,True,False,299,299,L,120.61,6fa5f511bf1a2019dde25ecbb2e8909e
Normal-10089.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10089.png,Normal/images/Normal-10089.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10089.png,Normal/masks/Normal-10089.png,True,False,299,299,L,114.76,b4d6591f2ccd56185b474609f17b30c3
Normal-1009.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1009.png,Normal/images/Normal-1009.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1009.png,Normal/masks/Normal-1009.png,True,False,299,299,L,111.77,86385606303bf783a47c259839323076
Normal-10090.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10090.png,Normal/images/Normal-10090.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10090.png,Normal/masks/Normal-10090.png,True,False,299,299,L,177.31,11a362b39f16732500bca20a93691018
Normal-10091.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10091.png,Normal/images/Normal-10091.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10091.png,Normal/masks/Normal-10091.png,True,False,299,299,L,109.8,d6c3fa7b86dbd031f7c5eda353f5c546
Normal-10092.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10092.png,Normal/images/Normal-10092.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10092.png,Normal/masks/Normal-10092.png,True,False,299,299,L,107.22,1caff7439f333990159842da660158ec
Normal-10093.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10093.png,Normal/images/Normal-10093.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10093.png,Normal/masks/Normal-10093.png,True,False,299,299,L,126.18,6f791ae31423efe150a52a59a1acb0a5
Normal-10094.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10094.png,Normal/images/Normal-10094.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10094.png,Normal/masks/Normal-10094.png,True,False,299,299,L,122.03,d63f1fa3769f522bac2430004c61bfdd
Normal-10095.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10095.png,Normal/images/Normal-10095.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10095.png,Normal/masks/Normal-10095.png,True,False,299,299,L,130.98,83a4c7062e3bb6e1fc4bc2a4a32ce821
Normal-10096.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10096.png,Normal/images/Normal-10096.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10096.png,Normal/masks/Normal-10096.png,True,False,299,299,L,139.78,91d9627d91337035e7c904abe8debf00
Normal-10097.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10097.png,Normal/images/Normal-10097.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10097.png,Normal/masks/Normal-10097.png,True,False,299,299,L,156.67,5f8b03d6cfdb0e94c37414bf91900a13
Normal-10098.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10098.png,Normal/images/Normal-10098.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10098.png,Normal/masks/Normal-10098.png,True,False,299,299,L,169.91,e24d9c2f619264ecd7d498db21546aeb
Normal-10099.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10099.png,Normal/images/Normal-10099.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10099.png,Normal/masks/Normal-10099.png,True,False,299,299,L,118.73,df52989d7167e34cc04dab153e09c803
Normal-101.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-101.png,Normal/images/Normal-101.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-101.png,Normal/masks/Normal-101.png,True,False,299,299,L,123.04,dc04e4391646a8472eadaf3ebefc8ed8
Normal-1010.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1010.png,Normal/images/Normal-1010.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1010.png,Normal/masks/Normal-1010.png,True,False,299,299,L,136.6,338f9442f0d89fb75bf4dbd8c85ac522
Normal-10100.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10100.png,Normal/images/Normal-10100.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10100.png,Normal/masks/Normal-10100.png,True,False,299,299,L,116.11,04f785f805c24a2120e7f2c815fdaf9d
Normal-10101.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10101.png,Normal/images/Normal-10101.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10101.png,Normal/masks/Normal-10101.png,True,False,299,299,L,108.44,e4e9e0559649c381a3bd60b089913c35
Normal-10102.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10102.png,Normal/images/Normal-10102.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10102.png,Normal/masks/Normal-10102.png,True,False,299,299,L,146.27,f6989a9f38c26d9880439fe74eea2528
Normal-10103.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10103.png,Normal/images/Normal-10103.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10103.png,Normal/masks/Normal-10103.png,True,False,299,299,L,124.2,f1288880b5ba3e9c608b2778d2cd4d9b
Normal-10104.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10104.png,Normal/images/Normal-10104.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10104.png,Normal/masks/Normal-10104.png,True,False,299,299,L,143.48,d94ecb896c2c62bd25a17c63e83bd747
Normal-10105.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10105.png,Normal/images/Normal-10105.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10105.png,Normal/masks/Normal-10105.png,True,False,299,299,L,104.27,f3a57c53815d1df6d312d5190dbad694
Normal-10106.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10106.png,Normal/images/Normal-10106.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10106.png,Normal/masks/Normal-10106.png,True,False,299,299,L,169.6,bcd4d0cd27f692ef0a7e44bd4847df5e
Normal-10107.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10107.png,Normal/images/Normal-10107.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10107.png,Normal/masks/Normal-10107.png,True,False,299,299,L,137.24,91483663b09e1b5133e4263e6f65e105
Normal-10108.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10108.png,Normal/images/Normal-10108.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10108.png,Normal/masks/Normal-10108.png,True,False,299,299,L,133.67,517eb2796c3451580214eeb47dfdfd72
Normal-10109.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10109.png,Normal/images/Normal-10109.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10109.png,Normal/masks/Normal-10109.png,True,False,299,299,L,174.17,8c46d9028be559d45e2bb11b55b788c4
Normal-1011.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1011.png,Normal/images/Normal-1011.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1011.png,Normal/masks/Normal-1011.png,True,False,299,299,L,125.14,3eaef0ebbbbf5fbd7a02629f999118f0
Normal-10110.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10110.png,Normal/images/Normal-10110.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10110.png,Normal/masks/Normal-10110.png,True,False,299,299,L,156.24,765d07a25a72fa1bcf08676d49069963
Normal-10111.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10111.png,Normal/images/Normal-10111.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10111.png,Normal/masks/Normal-10111.png,True,False,299,299,L,121.61,daded22bc9b1587554dc79de3d190a9a
Normal-10112.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10112.png,Normal/images/Normal-10112.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10112.png,Normal/masks/Normal-10112.png,True,False,299,299,L,155.99,9a7ed4911458f4f2bea5da1d20020352
Normal-10113.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10113.png,Normal/images/Normal-10113.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10113.png,Normal/masks/Normal-10113.png,True,False,299,299,L,114.53,38f9d2591c01747cae173dd1a12d42d3
Normal-10114.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10114.png,Normal/images/Normal-10114.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10114.png,Normal/masks/Normal-10114.png,True,False,299,299,L,136.5,52769e86f0a0614f4f362c973a8f5c80
Normal-10115.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10115.png,Normal/images/Normal-10115.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10115.png,Normal/masks/Normal-10115.png,True,False,299,299,L,114.95,17506c47ca0b8f0c734fc0fa8d958c6f
Normal-10116.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10116.png,Normal/images/Normal-10116.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10116.png,Normal/masks/Normal-10116.png,True,False,299,299,L,158.98,09ce464d0716c5f197b45574e88cb9a5
Normal-10117.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10117.png,Normal/images/Normal-10117.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10117.png,Normal/masks/Normal-10117.png,True,False,299,299,L,111.37,113aeb3ed5610a1e62bc2d89003de6a0
Normal-10118.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10118.png,Normal/images/Normal-10118.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10118.png,Normal/masks/Normal-10118.png,True,False,299,299,L,133.9,ddf04685390cc9ad593cd513490ccb98
Normal-10119.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10119.png,Normal/images/Normal-10119.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10119.png,Normal/masks/Normal-10119.png,True,False,299,299,L,116.77,94b4504cb619baf964c29707dd2136d7
Normal-1012.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1012.png,Normal/images/Normal-1012.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1012.png,Normal/masks/Normal-1012.png,True,False,299,299,L,114.95,f30a7a772cb2e5db906f964c381822af
Normal-10120.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10120.png,Normal/images/Normal-10120.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10120.png,Normal/masks/Normal-10120.png,True,False,299,299,L,160.18,81da9b48945c5ee11b3052bce4cf9f14
Normal-10121.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10121.png,Normal/images/Normal-10121.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10121.png,Normal/masks/Normal-10121.png,True,False,299,299,L,152.99,ef589ed810b61ca5edf380e876eb2b26
Normal-10122.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10122.png,Normal/images/Normal-10122.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10122.png,Normal/masks/Normal-10122.png,True,False,299,299,L,107.34,2ba58c95488d64588488b3f713bde9bb
Normal-10123.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10123.png,Normal/images/Normal-10123.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10123.png,Normal/masks/Normal-10123.png,True,False,299,299,L,119.82,df02cb5b18967c1c5fee9f7e36926d9e
Normal-10124.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10124.png,Normal/images/Normal-10124.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10124.png,Normal/masks/Normal-10124.png,True,False,299,299,L,157.65,671a8f036fe2a20471853f320ef293d8
Normal-10125.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10125.png,Normal/images/Normal-10125.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10125.png,Normal/masks/Normal-10125.png,True,False,299,299,L,170.27,8939f266e371e0f5de7c880643f95388
Normal-10126.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10126.png,Normal/images/Normal-10126.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10126.png,Normal/masks/Normal-10126.png,True,False,299,299,L,118.79,485aa3c130fb0f11c7ef697a3ef8bafa
Normal-10127.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10127.png,Normal/images/Normal-10127.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10127.png,Normal/masks/Normal-10127.png,True,False,299,299,L,116.04,5556a5181bc16cca5232613f096e9660
Normal-10128.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10128.png,Normal/images/Normal-10128.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10128.png,Normal/masks/Normal-10128.png,True,False,299,299,L,103.16,ee942b776474ece3ad5b275b68db59df
Normal-10129.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10129.png,Normal/images/Normal-10129.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10129.png,Normal/masks/Normal-10129.png,True,False,299,299,L,114.45,ae9323badad5e7b83f98f4ba356c3dde
Normal-1013.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1013.png,Normal/images/Normal-1013.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1013.png,Normal/masks/Normal-1013.png,True,False,299,299,L,115.4,130a09306d3211d2850076ee8788bae9
Normal-10130.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10130.png,Normal/images/Normal-10130.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10130.png,Normal/masks/Normal-10130.png,True,False,299,299,L,126.89,9734f431fac86c9d1dfa7d3deb343cd7
Normal-10131.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10131.png,Normal/images/Normal-10131.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10131.png,Normal/masks/Normal-10131.png,True,False,299,299,L,160.78,a16446f15b916a7b0f697f824eb9e0c8
Normal-10132.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10132.png,Normal/images/Normal-10132.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10132.png,Normal/masks/Normal-10132.png,True,False,299,299,L,128.5,4bc70621d1f2ebeba1f8b8eb3685509c
Normal-10133.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10133.png,Normal/images/Normal-10133.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10133.png,Normal/masks/Normal-10133.png,True,False,299,299,L,109.75,c1eee25c385e16d2604436120e6261a9
Normal-10134.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10134.png,Normal/images/Normal-10134.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10134.png,Normal/masks/Normal-10134.png,True,False,299,299,L,167.97,62a27c52eadc80a3685febe993227c6d
Normal-10135.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10135.png,Normal/images/Normal-10135.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10135.png,Normal/masks/Normal-10135.png,True,False,299,299,L,113.02,205a7b5614381452b5d81992d2a05803
Normal-10136.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10136.png,Normal/images/Normal-10136.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10136.png,Normal/masks/Normal-10136.png,True,False,299,299,L,124.07,ba09895b06e5468f52d8b0f9ff44ff6f
Normal-10137.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10137.png,Normal/images/Normal-10137.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10137.png,Normal/masks/Normal-10137.png,True,False,299,299,L,136.42,f5f85ba4dd898d155ff0f6a7fb2ebf02
Normal-10138.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10138.png,Normal/images/Normal-10138.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10138.png,Normal/masks/Normal-10138.png,True,False,299,299,L,111.44,16501fe6182b9a7f0b6c26221c43efd7
Normal-10139.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10139.png,Normal/images/Normal-10139.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10139.png,Normal/masks/Normal-10139.png,True,False,299,299,L,141.03,f32ca4d46d94724227424fa125c87a45
Normal-1014.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1014.png,Normal/images/Normal-1014.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1014.png,Normal/masks/Normal-1014.png,True,False,299,299,L,155.06,1bfe46990a74074935ebbbbc2c7266f5
Normal-10140.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10140.png,Normal/images/Normal-10140.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10140.png,Normal/masks/Normal-10140.png,True,False,299,299,L,153.04,64a5b5351e55d6fff92ae18eef1cac83
Normal-10141.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10141.png,Normal/images/Normal-10141.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10141.png,Normal/masks/Normal-10141.png,True,False,299,299,L,173.95,df03598d21487b65a110d23cd8793b72
Normal-10142.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10142.png,Normal/images/Normal-10142.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10142.png,Normal/masks/Normal-10142.png,True,False,299,299,L,135.24,e09ff2eecc6f24191385564e335019d1
Normal-10143.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10143.png,Normal/images/Normal-10143.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10143.png,Normal/masks/Normal-10143.png,True,False,299,299,L,120.49,2297f991a7c9de5b2b95b3567c71fbf4
Normal-10144.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10144.png,Normal/images/Normal-10144.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10144.png,Normal/masks/Normal-10144.png,True,False,299,299,L,107.9,2c2d24b92fa626d99e94dffc53c24114
Normal-10145.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10145.png,Normal/images/Normal-10145.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10145.png,Normal/masks/Normal-10145.png,True,False,299,299,L,115.03,47c96a95ea289cd5d8dca48dcea4aa82
Normal-10146.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10146.png,Normal/images/Normal-10146.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10146.png,Normal/masks/Normal-10146.png,True,False,299,299,L,136.97,4cbef5f612e26bb61a49f6900d7a3fbf
Normal-10147.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10147.png,Normal/images/Normal-10147.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10147.png,Normal/masks/Normal-10147.png,True,False,299,299,L,174.35,0b8321043d1dd22b274a48bb7ad1587f
Normal-10148.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10148.png,Normal/images/Normal-10148.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10148.png,Normal/masks/Normal-10148.png,True,False,299,299,L,121.6,23e69b264afab78527ac23859f93cda9
Normal-10149.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10149.png,Normal/images/Normal-10149.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10149.png,Normal/masks/Normal-10149.png,True,False,299,299,L,130.66,9e68859230b20c9c390b1cdabeaa0d98
Normal-1015.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1015.png,Normal/images/Normal-1015.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1015.png,Normal/masks/Normal-1015.png,True,False,299,299,L,117.79,9c44c5aa8718a634c9fe77a5ff9f626f
Normal-10150.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10150.png,Normal/images/Normal-10150.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10150.png,Normal/masks/Normal-10150.png,True,False,299,299,L,117.77,4a50e7d4427eac54aa43b100068382b6
Normal-10151.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10151.png,Normal/images/Normal-10151.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10151.png,Normal/masks/Normal-10151.png,True,False,299,299,L,116.01,dfe0144db2b99c9e3b5c9d70aa9770a2
Normal-10152.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10152.png,Normal/images/Normal-10152.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10152.png,Normal/masks/Normal-10152.png,True,False,299,299,L,132.63,4c8418cd31dfce036ed67c0790555824
Normal-10153.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10153.png,Normal/images/Normal-10153.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10153.png,Normal/masks/Normal-10153.png,True,False,299,299,L,117.56,26a87fc24e957f6e220f5ffb137f3a7c
Normal-10154.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10154.png,Normal/images/Normal-10154.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10154.png,Normal/masks/Normal-10154.png,True,False,299,299,L,132.86,494e09bf44a9238fbe7f8f2351271392
Normal-10155.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10155.png,Normal/images/Normal-10155.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10155.png,Normal/masks/Normal-10155.png,True,False,299,299,L,157.56,ca1e37230d2dd80d9bed56c8c442c35e
Normal-10156.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10156.png,Normal/images/Normal-10156.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10156.png,Normal/masks/Normal-10156.png,True,False,299,299,L,157.19,e16374d48bd98708bf66ed8f7b120990
Normal-10157.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10157.png,Normal/images/Normal-10157.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10157.png,Normal/masks/Normal-10157.png,True,False,299,299,L,135.86,3489d95362c6b83caf0e1df663f5d048
Normal-10158.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10158.png,Normal/images/Normal-10158.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10158.png,Normal/masks/Normal-10158.png,True,False,299,299,L,102.44,04577c9a66ea8f53fb32eb613b258bb3
Normal-10159.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10159.png,Normal/images/Normal-10159.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10159.png,Normal/masks/Normal-10159.png,True,False,299,299,L,121.98,19aafbd04891c88bf31108ef35326bcc
Normal-1016.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1016.png,Normal/images/Normal-1016.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1016.png,Normal/masks/Normal-1016.png,True,False,299,299,L,169.7,1a1004387666192df2710ec61d845f4f
Normal-10160.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10160.png,Normal/images/Normal-10160.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10160.png,Normal/masks/Normal-10160.png,True,False,299,299,L,157.68,81efac43a1b64369f1e882b85f64accf
Normal-10161.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10161.png,Normal/images/Normal-10161.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10161.png,Normal/masks/Normal-10161.png,True,False,299,299,L,174.48,48b3c0b61c51de684e49347758df18f9
Normal-10162.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10162.png,Normal/images/Normal-10162.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10162.png,Normal/masks/Normal-10162.png,True,False,299,299,L,128.25,053d2c7e3314f0539b433e2e278c6a8f
Normal-10163.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10163.png,Normal/images/Normal-10163.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10163.png,Normal/masks/Normal-10163.png,True,False,299,299,L,127.78,c7c017668430d003eba84c50e59025e8
Normal-10164.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10164.png,Normal/images/Normal-10164.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10164.png,Normal/masks/Normal-10164.png,True,False,299,299,L,117.25,ca357f2f551fbb42f6da6a0332b76387
Normal-10165.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10165.png,Normal/images/Normal-10165.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10165.png,Normal/masks/Normal-10165.png,True,False,299,299,L,174.5,f8816e936bfea7412d6fca42c8d57621
Normal-10166.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10166.png,Normal/images/Normal-10166.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10166.png,Normal/masks/Normal-10166.png,True,False,299,299,L,149.67,8906907e95db7ccaabfce45099498018
Normal-10167.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10167.png,Normal/images/Normal-10167.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10167.png,Normal/masks/Normal-10167.png,True,False,299,299,L,126.48,4e2c9c68c9d09a0998b17bedcf1d4a9b
Normal-10168.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10168.png,Normal/images/Normal-10168.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10168.png,Normal/masks/Normal-10168.png,True,False,299,299,L,121.72,fb97b7b2fef32ab7641858c30f34308d
Normal-10169.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10169.png,Normal/images/Normal-10169.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10169.png,Normal/masks/Normal-10169.png,True,False,299,299,L,179.82,694461c6f1c26c1827c99381ec144ee2
Normal-1017.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1017.png,Normal/images/Normal-1017.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1017.png,Normal/masks/Normal-1017.png,True,False,299,299,L,137.74,1e4a6171ff3cb7f6be895cdcbd3a5b07
Normal-10170.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10170.png,Normal/images/Normal-10170.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10170.png,Normal/masks/Normal-10170.png,True,False,299,299,L,191.83,7f3a3980ae75c946b647a124b0fdce14
Normal-10171.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10171.png,Normal/images/Normal-10171.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10171.png,Normal/masks/Normal-10171.png,True,False,299,299,L,163.62,629fad68bd191212e3a3da28190d17c2
Normal-10172.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10172.png,Normal/images/Normal-10172.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10172.png,Normal/masks/Normal-10172.png,True,False,299,299,L,160.82,4965d8075ff3587259e3b58b2cbdce81
Normal-10173.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10173.png,Normal/images/Normal-10173.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10173.png,Normal/masks/Normal-10173.png,True,False,299,299,L,144.24,fac4b8d599ff685e5456870b5ce4afe3
Normal-10174.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10174.png,Normal/images/Normal-10174.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10174.png,Normal/masks/Normal-10174.png,True,False,299,299,L,115.82,780836be0b1bb0242e2dcca91e854fe3
Normal-10175.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10175.png,Normal/images/Normal-10175.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10175.png,Normal/masks/Normal-10175.png,True,False,299,299,L,169.55,544df87c75fb6edcd370218512c74b8a
Normal-10176.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10176.png,Normal/images/Normal-10176.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10176.png,Normal/masks/Normal-10176.png,True,False,299,299,L,131.95,e421cad03c8778c1f5f2aaf0e02d0e00
Normal-10177.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10177.png,Normal/images/Normal-10177.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10177.png,Normal/masks/Normal-10177.png,True,False,299,299,L,124.76,d0b58e8f2813f7368a5f6dc7411c6db7
Normal-10178.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10178.png,Normal/images/Normal-10178.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10178.png,Normal/masks/Normal-10178.png,True,False,299,299,L,103.76,9a0ef23f4c6d4887fbc8bddaa0ed56a4
Normal-10179.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10179.png,Normal/images/Normal-10179.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10179.png,Normal/masks/Normal-10179.png,True,False,299,299,L,158.81,4139ad10eb729f09af2ea36d262f3d0d
Normal-1018.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1018.png,Normal/images/Normal-1018.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1018.png,Normal/masks/Normal-1018.png,True,False,299,299,L,121.49,375c641bcbc0f4b59c02c66a92810184
Normal-10180.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10180.png,Normal/images/Normal-10180.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10180.png,Normal/masks/Normal-10180.png,True,False,299,299,L,119.54,7ceb9b6d6bb5da181f6d80bd90f07784
Normal-10181.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10181.png,Normal/images/Normal-10181.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10181.png,Normal/masks/Normal-10181.png,True,False,299,299,L,131.77,97e712fdb20d7613016c37cfc0af1750
Normal-10182.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10182.png,Normal/images/Normal-10182.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10182.png,Normal/masks/Normal-10182.png,True,False,299,299,L,130.65,714e926a51670862dd478b15a03e97e2
Normal-10183.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10183.png,Normal/images/Normal-10183.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10183.png,Normal/masks/Normal-10183.png,True,False,299,299,L,121.61,52af01434fbb49e17af258b69af7f063
Normal-10184.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10184.png,Normal/images/Normal-10184.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10184.png,Normal/masks/Normal-10184.png,True,False,299,299,L,132.32,cb4498e5b3ec2a3a45dd366c59333125
Normal-10185.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10185.png,Normal/images/Normal-10185.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10185.png,Normal/masks/Normal-10185.png,True,False,299,299,L,131.78,6edfcf0a6e201a883759e8a8c5d25b7f
Normal-10186.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10186.png,Normal/images/Normal-10186.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10186.png,Normal/masks/Normal-10186.png,True,False,299,299,L,114.93,fb720935023f1d4a9ee3ef36aaf223b0
Normal-10187.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10187.png,Normal/images/Normal-10187.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10187.png,Normal/masks/Normal-10187.png,True,False,299,299,L,160.27,97cbd591509f0d8066367df7fa2dcb55
Normal-10188.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10188.png,Normal/images/Normal-10188.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10188.png,Normal/masks/Normal-10188.png,True,False,299,299,L,126.07,87f13b2df976e1bf3082b8493cb93911
Normal-10189.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10189.png,Normal/images/Normal-10189.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10189.png,Normal/masks/Normal-10189.png,True,False,299,299,L,121.46,2d33ae1550db04a0cfd25d70d7155ec4
Normal-1019.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1019.png,Normal/images/Normal-1019.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1019.png,Normal/masks/Normal-1019.png,True,False,299,299,L,160.48,7fe93d2520a1f8f4d97b5b380b71ae26
Normal-10190.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10190.png,Normal/images/Normal-10190.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10190.png,Normal/masks/Normal-10190.png,True,False,299,299,L,176.21,1a3b4ed7ad5cd1f5a0fcdc2b16cd6bfb
Normal-10191.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10191.png,Normal/images/Normal-10191.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10191.png,Normal/masks/Normal-10191.png,True,False,299,299,L,96.86,7b6a702e072412b7fb2937886c1c5605
Normal-10192.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-10192.png,Normal/images/Normal-10192.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-10192.png,Normal/masks/Normal-10192.png,True,False,299,299,L,129.7,409d891d99594648c90be900c10c8532
Normal-102.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-102.png,Normal/images/Normal-102.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-102.png,Normal/masks/Normal-102.png,True,False,299,299,L,135.78,cd820ef8511f0192d382ea03c1a92cf0
Normal-1020.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1020.png,Normal/images/Normal-1020.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1020.png,Normal/masks/Normal-1020.png,True,False,299,299,L,131.09,e093b67b34036b7b2217b606552d2405
Normal-1021.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1021.png,Normal/images/Normal-1021.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1021.png,Normal/masks/Normal-1021.png,True,False,299,299,L,126.17,c8191747d9e3578a480a60e63d3bcfff
Normal-1022.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1022.png,Normal/images/Normal-1022.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1022.png,Normal/masks/Normal-1022.png,True,False,299,299,L,117.38,b53a32dc7af34a78a4670eeaea8abfd2
Normal-1023.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1023.png,Normal/images/Normal-1023.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1023.png,Normal/masks/Normal-1023.png,True,False,299,299,L,127.3,7e83cc84d3faf50d69bfe310ea928e07
Normal-1024.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1024.png,Normal/images/Normal-1024.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1024.png,Normal/masks/Normal-1024.png,True,False,299,299,L,141.17,f9f6ffaf5948fa3c0809599b5a114eb8
Normal-1025.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1025.png,Normal/images/Normal-1025.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1025.png,Normal/masks/Normal-1025.png,True,False,299,299,L,134.27,a677b081dadeafc715d57f0bf8d45bdb
Normal-1026.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1026.png,Normal/images/Normal-1026.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1026.png,Normal/masks/Normal-1026.png,True,False,299,299,L,157.3,17525fbb40905975e210af17a21a2647
Normal-1027.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1027.png,Normal/images/Normal-1027.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1027.png,Normal/masks/Normal-1027.png,True,False,299,299,L,139.23,69b82d699e9e038e811b3a6cf8f3fb2e
Normal-1028.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1028.png,Normal/images/Normal-1028.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1028.png,Normal/masks/Normal-1028.png,True,False,299,299,L,116.15,c7f77d4e0340090551255f00516a39f4
Normal-1029.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1029.png,Normal/images/Normal-1029.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1029.png,Normal/masks/Normal-1029.png,True,False,299,299,L,138.33,926780d66d1f376797d3a26d6cfa5270
Normal-103.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-103.png,Normal/images/Normal-103.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-103.png,Normal/masks/Normal-103.png,True,False,299,299,L,111.99,94805baa4f5e28a5460d262bb946f831
Normal-1030.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1030.png,Normal/images/Normal-1030.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1030.png,Normal/masks/Normal-1030.png,True,False,299,299,L,137.33,76951d73a20110d8da8354adc661e8e2
Normal-1031.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1031.png,Normal/images/Normal-1031.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1031.png,Normal/masks/Normal-1031.png,True,False,299,299,L,140.44,ab773df95fd11595a3ba0cc55330ae80
Normal-1032.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1032.png,Normal/images/Normal-1032.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1032.png,Normal/masks/Normal-1032.png,True,False,299,299,L,135.86,aaad59209dce1cc5955bd6bbe72b0598
Normal-1033.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1033.png,Normal/images/Normal-1033.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1033.png,Normal/masks/Normal-1033.png,True,False,299,299,L,122.81,842028d3248421b8d15e9c798d7f1f0c
Normal-1034.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1034.png,Normal/images/Normal-1034.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1034.png,Normal/masks/Normal-1034.png,True,False,299,299,L,125.48,432f941d4d1bfdec9e7b0a95197fbe90
Normal-1035.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1035.png,Normal/images/Normal-1035.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1035.png,Normal/masks/Normal-1035.png,True,False,299,299,L,109.49,441e320a043a5f7f2ca7d49b21965ef4
Normal-1036.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1036.png,Normal/images/Normal-1036.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1036.png,Normal/masks/Normal-1036.png,True,False,299,299,L,101.8,8b073bce64e4fcf837a1e1374af6f277
Normal-1037.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1037.png,Normal/images/Normal-1037.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1037.png,Normal/masks/Normal-1037.png,True,False,299,299,L,98.08,60cac0a15262f65fcdcdfa3d89643fcd
Normal-1038.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1038.png,Normal/images/Normal-1038.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1038.png,Normal/masks/Normal-1038.png,True,False,299,299,L,107.24,d0d1d12cd691b006c8f6d4f135bd3d3b
Normal-1039.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1039.png,Normal/images/Normal-1039.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1039.png,Normal/masks/Normal-1039.png,True,False,299,299,L,124.43,eaece1ac605b5d132d5f8d77e918b9a7
Normal-104.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-104.png,Normal/images/Normal-104.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-104.png,Normal/masks/Normal-104.png,True,False,299,299,L,131.74,c77e6c237c49ed5e4b179b9668d2f6d0
Normal-1040.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1040.png,Normal/images/Normal-1040.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1040.png,Normal/masks/Normal-1040.png,True,False,299,299,L,113.99,56f2d55aefd525249c0815db4222907a
Normal-1041.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1041.png,Normal/images/Normal-1041.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1041.png,Normal/masks/Normal-1041.png,True,False,299,299,L,115.83,2e534273f62e2238df0f1c43a5012766
Normal-1042.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1042.png,Normal/images/Normal-1042.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1042.png,Normal/masks/Normal-1042.png,True,False,299,299,L,121.62,3037ec63c07159d02740c2ec9a868fb0
Normal-1043.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1043.png,Normal/images/Normal-1043.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1043.png,Normal/masks/Normal-1043.png,True,False,299,299,L,102.62,40ab03581aa2e4f619278cdf868e99aa
Normal-1044.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1044.png,Normal/images/Normal-1044.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1044.png,Normal/masks/Normal-1044.png,True,False,299,299,L,112.82,2d1a0e5a6fc512586e5f1aa304bc228f
Normal-1045.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1045.png,Normal/images/Normal-1045.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1045.png,Normal/masks/Normal-1045.png,True,False,299,299,L,119.09,5f97df44c168d840ad368d2d70bd2156
Normal-1046.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1046.png,Normal/images/Normal-1046.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1046.png,Normal/masks/Normal-1046.png,True,False,299,299,L,120.34,21cfe77699654b7f8b332ba98245e3a6
Normal-1047.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1047.png,Normal/images/Normal-1047.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1047.png,Normal/masks/Normal-1047.png,True,False,299,299,L,103.17,df9099f531a1dbc8a12bf907e7386549
Normal-1048.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1048.png,Normal/images/Normal-1048.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1048.png,Normal/masks/Normal-1048.png,True,False,299,299,L,122.81,a787fe8e3ecb30675e483246dd97fdf6
Normal-1049.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1049.png,Normal/images/Normal-1049.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1049.png,Normal/masks/Normal-1049.png,True,False,299,299,L,125.19,ce386bbb2d38203de264b3276adad799
Normal-105.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-105.png,Normal/images/Normal-105.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-105.png,Normal/masks/Normal-105.png,True,False,299,299,L,121.33,98efa59ae27113df893738df5d904adf
Normal-1050.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1050.png,Normal/images/Normal-1050.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1050.png,Normal/masks/Normal-1050.png,True,False,299,299,L,141.9,bef2014ccd880ec78557bec6da43be75
Normal-1051.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1051.png,Normal/images/Normal-1051.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1051.png,Normal/masks/Normal-1051.png,True,False,299,299,L,124.02,64d6afcb04b203c80c4317ce025ce969
Normal-1052.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1052.png,Normal/images/Normal-1052.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1052.png,Normal/masks/Normal-1052.png,True,False,299,299,L,135.71,68ab4239612ec217c17ebf922df13e1f
Normal-1053.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1053.png,Normal/images/Normal-1053.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1053.png,Normal/masks/Normal-1053.png,True,False,299,299,L,149.19,726fa31452cd5b399ad5d22dfeff2e7d
Normal-1054.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1054.png,Normal/images/Normal-1054.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1054.png,Normal/masks/Normal-1054.png,True,False,299,299,L,111.31,71f4958e31804f860f2d01037dfd0c00
Normal-1055.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1055.png,Normal/images/Normal-1055.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1055.png,Normal/masks/Normal-1055.png,True,False,299,299,L,130.39,1af043086af7289c7f850742c56fc9f8
Normal-1056.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1056.png,Normal/images/Normal-1056.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1056.png,Normal/masks/Normal-1056.png,True,False,299,299,L,111.69,b275ab2479fe26b8b805d29c32e73dba
Normal-1057.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1057.png,Normal/images/Normal-1057.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1057.png,Normal/masks/Normal-1057.png,True,False,299,299,L,130.92,409dd5a80ea9286c75109fd85a96d4a2
Normal-1058.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1058.png,Normal/images/Normal-1058.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1058.png,Normal/masks/Normal-1058.png,True,False,299,299,L,94.1,23d77f094140b9a12d2749b9c952aaeb
Normal-1059.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1059.png,Normal/images/Normal-1059.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1059.png,Normal/masks/Normal-1059.png,True,False,299,299,L,108.82,f1e6bf9b9cfd92ec7479655e22c40269
Normal-106.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-106.png,Normal/images/Normal-106.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-106.png,Normal/masks/Normal-106.png,True,False,299,299,L,102.91,630a054de13da2729070b5486526c22d
Normal-1060.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1060.png,Normal/images/Normal-1060.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1060.png,Normal/masks/Normal-1060.png,True,False,299,299,L,131.02,00043fc8a5db773ce8f4046bb7d816b2
Normal-1061.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1061.png,Normal/images/Normal-1061.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1061.png,Normal/masks/Normal-1061.png,True,False,299,299,L,106.03,2c84f1daae684f8ec88bc3f1546b0989
Normal-1062.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1062.png,Normal/images/Normal-1062.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1062.png,Normal/masks/Normal-1062.png,True,False,299,299,L,119.14,03c74df0915315de4c240a4c3a504dde
Normal-1063.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1063.png,Normal/images/Normal-1063.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1063.png,Normal/masks/Normal-1063.png,True,False,299,299,L,118.87,47dc04b7665fc5fc76758f783e829b06
Normal-1064.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1064.png,Normal/images/Normal-1064.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1064.png,Normal/masks/Normal-1064.png,True,False,299,299,L,143.56,abb501c8b646c54d5725c6fe07ce9b00
Normal-1065.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1065.png,Normal/images/Normal-1065.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1065.png,Normal/masks/Normal-1065.png,True,False,299,299,L,113.79,163fd7bfef0b73eed0bdb44a3f42fc99
Normal-1066.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1066.png,Normal/images/Normal-1066.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1066.png,Normal/masks/Normal-1066.png,True,False,299,299,L,122.9,b9ec660af6530fa4fe0824738d7bfedf
Normal-1067.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1067.png,Normal/images/Normal-1067.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1067.png,Normal/masks/Normal-1067.png,True,False,299,299,L,133.51,bd71259aabaa29ec13fbecbc0d56f661
Normal-1068.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1068.png,Normal/images/Normal-1068.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1068.png,Normal/masks/Normal-1068.png,True,False,299,299,L,126.95,2bdd796c9c3507f41593990772d12ea8
Normal-1069.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1069.png,Normal/images/Normal-1069.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1069.png,Normal/masks/Normal-1069.png,True,False,299,299,L,129.82,51af310b20b59c980ba0e2307e8b7f24
Normal-107.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-107.png,Normal/images/Normal-107.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-107.png,Normal/masks/Normal-107.png,True,False,299,299,L,121.53,391e5481307e5cf55bb08efcf93a95d2
Normal-1070.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1070.png,Normal/images/Normal-1070.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1070.png,Normal/masks/Normal-1070.png,True,False,299,299,L,113.17,690e18b613bf22fe44d955a04eee6f8f
Normal-1071.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1071.png,Normal/images/Normal-1071.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1071.png,Normal/masks/Normal-1071.png,True,False,299,299,L,119.98,03d3f7aa36d49d5b8361078999911acb
Normal-1072.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1072.png,Normal/images/Normal-1072.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1072.png,Normal/masks/Normal-1072.png,True,False,299,299,L,131.74,8d8bbaa9e3c6e0f912f92d4821e6dba5
Normal-1073.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1073.png,Normal/images/Normal-1073.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1073.png,Normal/masks/Normal-1073.png,True,False,299,299,L,129.94,c5f35934ce072c2dd179c80d9b714119
Normal-1074.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1074.png,Normal/images/Normal-1074.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1074.png,Normal/masks/Normal-1074.png,True,False,299,299,L,120.38,1ec49fee0725f40c1ba92da8420737fb
Normal-1075.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1075.png,Normal/images/Normal-1075.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1075.png,Normal/masks/Normal-1075.png,True,False,299,299,L,103.6,782a5cbe44b0dda221fed03b9860bb34
Normal-1076.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1076.png,Normal/images/Normal-1076.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1076.png,Normal/masks/Normal-1076.png,True,False,299,299,L,107.33,4b40fa4a64eee19de6e9ac8547673f8b
Normal-1077.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1077.png,Normal/images/Normal-1077.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1077.png,Normal/masks/Normal-1077.png,True,False,299,299,L,131.8,4faece82692e1df5e421810ace9df98c
Normal-1078.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1078.png,Normal/images/Normal-1078.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1078.png,Normal/masks/Normal-1078.png,True,False,299,299,L,97.19,8e4f635d6b45c103d92cea0002062936
Normal-1079.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1079.png,Normal/images/Normal-1079.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1079.png,Normal/masks/Normal-1079.png,True,False,299,299,L,94.01,56c77f0a7adda87b6b133fb7ebd4e3ad
Normal-108.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-108.png,Normal/images/Normal-108.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-108.png,Normal/masks/Normal-108.png,True,False,299,299,L,104.71,6f72856e859e34883cf9e90dae6d254b
Normal-1080.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1080.png,Normal/images/Normal-1080.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1080.png,Normal/masks/Normal-1080.png,True,False,299,299,L,114.28,edf1e61ba6db13c145dd53a2f0953ccb
Normal-1081.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1081.png,Normal/images/Normal-1081.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1081.png,Normal/masks/Normal-1081.png,True,False,299,299,L,116.66,1b002f22b48e8895f49b558bb96e28c9
Normal-1082.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1082.png,Normal/images/Normal-1082.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1082.png,Normal/masks/Normal-1082.png,True,False,299,299,L,122.0,7845ccdf74d156d5d4ec430b07a241c1
Normal-1083.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1083.png,Normal/images/Normal-1083.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1083.png,Normal/masks/Normal-1083.png,True,False,299,299,L,115.22,d0883bc2580568212ef4286f121e65a1
Normal-1084.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1084.png,Normal/images/Normal-1084.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1084.png,Normal/masks/Normal-1084.png,True,False,299,299,L,117.32,364e3ab6a1ee4aec58f4fc2d5b577dfa
Normal-1085.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1085.png,Normal/images/Normal-1085.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1085.png,Normal/masks/Normal-1085.png,True,False,299,299,L,124.18,69a27dbcd91cab3bd30a01d3abadbc0e
Normal-1086.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1086.png,Normal/images/Normal-1086.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1086.png,Normal/masks/Normal-1086.png,True,False,299,299,L,118.05,92fef3813b2491d5e4ca3704cbdb2fb0
Normal-1087.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1087.png,Normal/images/Normal-1087.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1087.png,Normal/masks/Normal-1087.png,True,False,299,299,L,111.36,30ab5355374b3bba7c5a00d3d6564a12
Normal-1088.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1088.png,Normal/images/Normal-1088.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1088.png,Normal/masks/Normal-1088.png,True,False,299,299,L,131.35,10479332a5c8637ea3bf887fff54ba06
Normal-1089.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1089.png,Normal/images/Normal-1089.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1089.png,Normal/masks/Normal-1089.png,True,False,299,299,L,124.03,2e9e3a9c80eb087e3e1eabf507164f54
Normal-109.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-109.png,Normal/images/Normal-109.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-109.png,Normal/masks/Normal-109.png,True,False,299,299,L,117.35,ed0e168a631f66210a2a426b6eb1ad8a
Normal-1090.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1090.png,Normal/images/Normal-1090.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1090.png,Normal/masks/Normal-1090.png,True,False,299,299,L,128.03,7fb637482d949fa8258220ed27110c01
Normal-1091.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1091.png,Normal/images/Normal-1091.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1091.png,Normal/masks/Normal-1091.png,True,False,299,299,L,129.95,6e9a7df5856c314e329c6d29a10bbeb0
Normal-1092.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1092.png,Normal/images/Normal-1092.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1092.png,Normal/masks/Normal-1092.png,True,False,299,299,L,130.14,f12d36a22ab8b1674749a8a7d24a999e
Normal-1093.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1093.png,Normal/images/Normal-1093.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1093.png,Normal/masks/Normal-1093.png,True,False,299,299,L,150.01,bd394a2d0629fa7e2caad79b7865396a
Normal-1094.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1094.png,Normal/images/Normal-1094.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1094.png,Normal/masks/Normal-1094.png,True,False,299,299,L,143.96,90d427af30e888c16a90ad7320a07416
Normal-1095.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1095.png,Normal/images/Normal-1095.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1095.png,Normal/masks/Normal-1095.png,True,False,299,299,L,135.06,d015a5f0724a706c86a0a7cb09e32e98
Normal-1096.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1096.png,Normal/images/Normal-1096.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1096.png,Normal/masks/Normal-1096.png,True,False,299,299,L,131.2,f22be774f70b041cb844dfdd5d3629a2
Normal-1097.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1097.png,Normal/images/Normal-1097.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1097.png,Normal/masks/Normal-1097.png,True,False,299,299,L,108.13,8198d1d9737cbdfa5c0dee681276163c
Normal-1098.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1098.png,Normal/images/Normal-1098.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1098.png,Normal/masks/Normal-1098.png,True,False,299,299,L,94.17,9b66adc1c3be9153301dd3869a5836f9
Normal-1099.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1099.png,Normal/images/Normal-1099.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1099.png,Normal/masks/Normal-1099.png,True,False,299,299,L,106.99,a2501a388fe97b54d447d86ee1456d23
Normal-11.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-11.png,Normal/images/Normal-11.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-11.png,Normal/masks/Normal-11.png,True,False,299,299,L,141.43,b01370644f490c151ca05b25adf7dd2f
Normal-110.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-110.png,Normal/images/Normal-110.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-110.png,Normal/masks/Normal-110.png,True,False,299,299,L,112.83,94eecdd7a35a22dc39575943c11d9bd0
Normal-1100.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1100.png,Normal/images/Normal-1100.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1100.png,Normal/masks/Normal-1100.png,True,False,299,299,L,133.48,5265d5ba1f40f80434a616b62a5625e1
Normal-1101.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1101.png,Normal/images/Normal-1101.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1101.png,Normal/masks/Normal-1101.png,True,False,299,299,L,93.44,d89fbc69fb9e1422a85add363e4b75f1
Normal-1102.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1102.png,Normal/images/Normal-1102.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1102.png,Normal/masks/Normal-1102.png,True,False,299,299,L,129.79,8bed3ef6ea1c2dd52625eb5bd303adbd
Normal-1103.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1103.png,Normal/images/Normal-1103.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1103.png,Normal/masks/Normal-1103.png,True,False,299,299,L,129.79,e6c51bac70882b98034619d85d50ccec
Normal-1104.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1104.png,Normal/images/Normal-1104.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1104.png,Normal/masks/Normal-1104.png,True,False,299,299,L,123.13,a7082c6a09ff36ef665bd592677663dc
Normal-1105.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1105.png,Normal/images/Normal-1105.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1105.png,Normal/masks/Normal-1105.png,True,False,299,299,L,99.49,7d505270b3c24ab75a151bbb973d61c0
Normal-1106.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1106.png,Normal/images/Normal-1106.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1106.png,Normal/masks/Normal-1106.png,True,False,299,299,L,129.5,4bd720a1a642b3cd81ca2a64b7b50f4e
Normal-1107.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1107.png,Normal/images/Normal-1107.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1107.png,Normal/masks/Normal-1107.png,True,False,299,299,L,116.6,c422f4b40da6f75c135e9eefaece28f8
Normal-1108.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1108.png,Normal/images/Normal-1108.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1108.png,Normal/masks/Normal-1108.png,True,False,299,299,L,130.42,72bd307d5dbeddc54338c77eeb6778ac
Normal-1109.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1109.png,Normal/images/Normal-1109.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1109.png,Normal/masks/Normal-1109.png,True,False,299,299,L,108.72,ac6dc05577d2d74c2dc3989e083e9c6e
Normal-111.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-111.png,Normal/images/Normal-111.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-111.png,Normal/masks/Normal-111.png,True,False,299,299,L,126.33,a4efb144a6c4d3a4d829e7de3146200a
Normal-1110.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1110.png,Normal/images/Normal-1110.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1110.png,Normal/masks/Normal-1110.png,True,False,299,299,L,129.96,3ee26671965c26a82c8faf43e0cf63f9
Normal-1111.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1111.png,Normal/images/Normal-1111.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1111.png,Normal/masks/Normal-1111.png,True,False,299,299,L,114.01,1005e020fd384bbfef4852cc5f418357
Normal-1112.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1112.png,Normal/images/Normal-1112.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1112.png,Normal/masks/Normal-1112.png,True,False,299,299,L,142.94,1e2cc545172ef00752270db33b5ec3d7
Normal-1113.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1113.png,Normal/images/Normal-1113.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1113.png,Normal/masks/Normal-1113.png,True,False,299,299,L,104.69,12cfebbea25a851572010507844bd75d
Normal-1114.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1114.png,Normal/images/Normal-1114.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1114.png,Normal/masks/Normal-1114.png,True,False,299,299,L,128.24,20c7cd2ad77f0c0b9374880686dd234b
Normal-1115.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1115.png,Normal/images/Normal-1115.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1115.png,Normal/masks/Normal-1115.png,True,False,299,299,L,105.22,49a07446bc46cf3567a165c482b44d1c
Normal-1116.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1116.png,Normal/images/Normal-1116.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1116.png,Normal/masks/Normal-1116.png,True,False,299,299,L,124.16,9d264abb07ceccaaec8f533c109ebacc
Normal-1117.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1117.png,Normal/images/Normal-1117.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1117.png,Normal/masks/Normal-1117.png,True,False,299,299,L,125.51,9c30188f13ab6454e48abd9c233362d4
Normal-1118.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1118.png,Normal/images/Normal-1118.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1118.png,Normal/masks/Normal-1118.png,True,False,299,299,L,147.83,9979ee99da391eb85c94446c3b44a199
Normal-1119.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1119.png,Normal/images/Normal-1119.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1119.png,Normal/masks/Normal-1119.png,True,False,299,299,L,112.71,99821e669572b22d3d9fb99a4c34facc
Normal-112.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-112.png,Normal/images/Normal-112.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-112.png,Normal/masks/Normal-112.png,True,False,299,299,L,110.8,8360991684008da410745b0d1ac119d7
Normal-1120.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1120.png,Normal/images/Normal-1120.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1120.png,Normal/masks/Normal-1120.png,True,False,299,299,L,98.38,414aa0f80962e3a1dc86d7cdbb615564
Normal-1121.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1121.png,Normal/images/Normal-1121.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1121.png,Normal/masks/Normal-1121.png,True,False,299,299,L,122.84,7f649a551f64906e2ffc0195e64a4760
Normal-1122.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1122.png,Normal/images/Normal-1122.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1122.png,Normal/masks/Normal-1122.png,True,False,299,299,L,109.2,a5c0a2e3ec26f3435eaf59215b12a3ef
Normal-1123.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1123.png,Normal/images/Normal-1123.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1123.png,Normal/masks/Normal-1123.png,True,False,299,299,L,106.78,87345b45211003ea4b272309e930507b
Normal-1124.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1124.png,Normal/images/Normal-1124.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1124.png,Normal/masks/Normal-1124.png,True,False,299,299,L,114.25,92623719ea24551c1ccd954bc57500be
Normal-1125.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1125.png,Normal/images/Normal-1125.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1125.png,Normal/masks/Normal-1125.png,True,False,299,299,L,113.71,706b2f7a942c5e6257d241ed3f96623f
Normal-1126.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1126.png,Normal/images/Normal-1126.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1126.png,Normal/masks/Normal-1126.png,True,False,299,299,L,121.44,0d946c53e0fb8f0f345d6a4bad1c465e
Normal-1127.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1127.png,Normal/images/Normal-1127.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1127.png,Normal/masks/Normal-1127.png,True,False,299,299,L,128.01,dc18be8daebac59825498c85d6590a85
Normal-1128.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1128.png,Normal/images/Normal-1128.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1128.png,Normal/masks/Normal-1128.png,True,False,299,299,L,147.69,217e8ca41aa1b99bcbef8400ccbb4c44
Normal-1129.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1129.png,Normal/images/Normal-1129.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1129.png,Normal/masks/Normal-1129.png,True,False,299,299,L,127.53,cc0ce946fbbb3382bc48d79945c89c1f
Normal-113.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-113.png,Normal/images/Normal-113.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-113.png,Normal/masks/Normal-113.png,True,False,299,299,L,122.22,adef4882d9375ca2612d066957a1b218
Normal-1130.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1130.png,Normal/images/Normal-1130.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1130.png,Normal/masks/Normal-1130.png,True,False,299,299,L,99.49,145bc7056b979aced1c3035dbf32c6e8
Normal-1131.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1131.png,Normal/images/Normal-1131.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1131.png,Normal/masks/Normal-1131.png,True,False,299,299,L,114.68,7baf5dc2601ca41b41984c84dfdb5d99
Normal-1132.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1132.png,Normal/images/Normal-1132.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1132.png,Normal/masks/Normal-1132.png,True,False,299,299,L,118.44,c4aec2dfef7a1619ac1aba406c1efa36
Normal-1133.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1133.png,Normal/images/Normal-1133.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1133.png,Normal/masks/Normal-1133.png,True,False,299,299,L,104.18,ee282606808603e545c13b5580cd581c
Normal-1134.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1134.png,Normal/images/Normal-1134.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1134.png,Normal/masks/Normal-1134.png,True,False,299,299,L,127.59,69bc5dcd309764b1b07ad58d6bc6621b
Normal-1135.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1135.png,Normal/images/Normal-1135.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1135.png,Normal/masks/Normal-1135.png,True,False,299,299,L,127.41,ac55880938acc801910f75227c8a26fb
Normal-1136.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1136.png,Normal/images/Normal-1136.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1136.png,Normal/masks/Normal-1136.png,True,False,299,299,L,119.91,f3c69523fa2aae7151d0bb20634b7688
Normal-1137.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1137.png,Normal/images/Normal-1137.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1137.png,Normal/masks/Normal-1137.png,True,False,299,299,L,102.15,f4983b3e2bb19b37108e592b2cf3a300
Normal-1138.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1138.png,Normal/images/Normal-1138.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1138.png,Normal/masks/Normal-1138.png,True,False,299,299,L,128.44,3120da4b349ea99539b41bb3c2e06d68
Normal-1139.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1139.png,Normal/images/Normal-1139.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1139.png,Normal/masks/Normal-1139.png,True,False,299,299,L,154.55,2214b747f3a068f6b28e0d1ee921993b
Normal-114.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-114.png,Normal/images/Normal-114.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-114.png,Normal/masks/Normal-114.png,True,False,299,299,L,120.42,fb98d4976a430be69b4e2853295fa05c
Normal-1140.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1140.png,Normal/images/Normal-1140.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1140.png,Normal/masks/Normal-1140.png,True,False,299,299,L,120.96,188c7db32070695149f8420994be5818
Normal-1141.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1141.png,Normal/images/Normal-1141.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1141.png,Normal/masks/Normal-1141.png,True,False,299,299,L,157.44,256a21aa853995669241128f7140f003
Normal-1142.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1142.png,Normal/images/Normal-1142.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1142.png,Normal/masks/Normal-1142.png,True,False,299,299,L,108.53,e0f7ce8efdf434a0ce11fc0d6f5863de
Normal-1143.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1143.png,Normal/images/Normal-1143.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1143.png,Normal/masks/Normal-1143.png,True,False,299,299,L,115.75,a67cbee8bcda599684e29bd50a010ac9
Normal-1144.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1144.png,Normal/images/Normal-1144.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1144.png,Normal/masks/Normal-1144.png,True,False,299,299,L,137.51,16bd2b08323ea87d942d3442974f5b7e
Normal-1145.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1145.png,Normal/images/Normal-1145.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1145.png,Normal/masks/Normal-1145.png,True,False,299,299,L,115.72,1f0140feda8fff2e78ab59c6daf20c67
Normal-1146.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1146.png,Normal/images/Normal-1146.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1146.png,Normal/masks/Normal-1146.png,True,False,299,299,L,114.94,115ef4ad330e21fd1290c7f33b2e1149
Normal-1147.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1147.png,Normal/images/Normal-1147.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1147.png,Normal/masks/Normal-1147.png,True,False,299,299,L,107.73,eef6bec26af8d2fa95cf778e926c6334
Normal-1148.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1148.png,Normal/images/Normal-1148.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1148.png,Normal/masks/Normal-1148.png,True,False,299,299,L,121.29,3282a22cd33d5803665e8d91c5caee60
Normal-1149.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1149.png,Normal/images/Normal-1149.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1149.png,Normal/masks/Normal-1149.png,True,False,299,299,L,143.42,86a76de34fedc1e0a2a5b2034ffb919c
Normal-115.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-115.png,Normal/images/Normal-115.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-115.png,Normal/masks/Normal-115.png,True,False,299,299,L,129.22,c0bab67f7e7abac22548d22c36110b55
Normal-1150.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1150.png,Normal/images/Normal-1150.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1150.png,Normal/masks/Normal-1150.png,True,False,299,299,L,135.98,d7321d423b183c9e86e4aa72bdf19e44
Normal-1151.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1151.png,Normal/images/Normal-1151.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1151.png,Normal/masks/Normal-1151.png,True,False,299,299,L,134.71,27f88522fc2d9518da3fce9ab54df5c4
Normal-1152.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1152.png,Normal/images/Normal-1152.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1152.png,Normal/masks/Normal-1152.png,True,False,299,299,L,116.96,7a77de31c8800c176b7df8dfe70b1b74
Normal-1153.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1153.png,Normal/images/Normal-1153.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1153.png,Normal/masks/Normal-1153.png,True,False,299,299,L,133.48,825549f2cafb1981d452bb88ad870fd3
Normal-1154.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1154.png,Normal/images/Normal-1154.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1154.png,Normal/masks/Normal-1154.png,True,False,299,299,L,139.02,3bb524e11f2e390f37b7549f5254d171
Normal-1155.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1155.png,Normal/images/Normal-1155.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1155.png,Normal/masks/Normal-1155.png,True,False,299,299,L,123.51,866bf410c3e2c271ca9ccb83bb51af87
Normal-1156.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1156.png,Normal/images/Normal-1156.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1156.png,Normal/masks/Normal-1156.png,True,False,299,299,L,114.66,f27157960d5e7703092f21b3ac39a3c6
Normal-1157.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1157.png,Normal/images/Normal-1157.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1157.png,Normal/masks/Normal-1157.png,True,False,299,299,L,120.52,8c2e72d0a69dcf190d1683871136723b
Normal-1158.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1158.png,Normal/images/Normal-1158.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1158.png,Normal/masks/Normal-1158.png,True,False,299,299,L,128.94,d3b0c7e2824531be698482dafd1e0708
Normal-1159.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1159.png,Normal/images/Normal-1159.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1159.png,Normal/masks/Normal-1159.png,True,False,299,299,L,100.58,20d030dd8ae754b80dc83d789569c82b
Normal-116.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-116.png,Normal/images/Normal-116.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-116.png,Normal/masks/Normal-116.png,True,False,299,299,L,109.67,96149da06050a1fcc4320582068144b7
Normal-1160.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1160.png,Normal/images/Normal-1160.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1160.png,Normal/masks/Normal-1160.png,True,False,299,299,L,130.05,5d4df8546501bf8aab90aa0093653c82
Normal-1161.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1161.png,Normal/images/Normal-1161.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1161.png,Normal/masks/Normal-1161.png,True,False,299,299,L,133.59,22e71b6fc407a1b24bee04fdb7a025ef
Normal-1162.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1162.png,Normal/images/Normal-1162.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1162.png,Normal/masks/Normal-1162.png,True,False,299,299,L,116.28,57950348f39727f612bc6d7bc9709f32
Normal-1163.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1163.png,Normal/images/Normal-1163.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1163.png,Normal/masks/Normal-1163.png,True,False,299,299,L,90.4,6dc523cd9288d946951de55b0a540f43
Normal-1164.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1164.png,Normal/images/Normal-1164.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1164.png,Normal/masks/Normal-1164.png,True,False,299,299,L,109.95,5f6b3de53b604aa35162bf787e283220
Normal-1165.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1165.png,Normal/images/Normal-1165.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1165.png,Normal/masks/Normal-1165.png,True,False,299,299,L,130.77,59572b23ae476bc32a91e0d01228810e
Normal-1166.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1166.png,Normal/images/Normal-1166.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1166.png,Normal/masks/Normal-1166.png,True,False,299,299,L,121.19,22e9ce9e2ddb74d3156799ba49bf0fce
Normal-1167.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1167.png,Normal/images/Normal-1167.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1167.png,Normal/masks/Normal-1167.png,True,False,299,299,L,123.09,1e5e788104eaf6b354f6e737c6c169ec
Normal-1168.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1168.png,Normal/images/Normal-1168.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1168.png,Normal/masks/Normal-1168.png,True,False,299,299,L,118.25,5f710b7bd0c9269e1f8d857548e312ed
Normal-1169.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1169.png,Normal/images/Normal-1169.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1169.png,Normal/masks/Normal-1169.png,True,False,299,299,L,131.71,e1ad18bcdd1d8d129a791da053c8ce34
Normal-117.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-117.png,Normal/images/Normal-117.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-117.png,Normal/masks/Normal-117.png,True,False,299,299,L,108.23,684119c20ffad430e155212473f3ea05
Normal-1170.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1170.png,Normal/images/Normal-1170.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1170.png,Normal/masks/Normal-1170.png,True,False,299,299,L,140.27,353c1a9e49441444ba5c3154c6f829e5
Normal-1171.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1171.png,Normal/images/Normal-1171.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1171.png,Normal/masks/Normal-1171.png,True,False,299,299,L,125.6,90af0b011eabbb2a71fd93f8e50abc92
Normal-1172.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1172.png,Normal/images/Normal-1172.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1172.png,Normal/masks/Normal-1172.png,True,False,299,299,L,118.14,1137d7bc53fd906127c25a7840f82f9e
Normal-1173.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1173.png,Normal/images/Normal-1173.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1173.png,Normal/masks/Normal-1173.png,True,False,299,299,L,116.7,19a59f6d2f67bb55b36a59d11e621d86
Normal-1174.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1174.png,Normal/images/Normal-1174.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1174.png,Normal/masks/Normal-1174.png,True,False,299,299,L,153.65,fb08280be7f92004bb34377020d13229
Normal-1175.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1175.png,Normal/images/Normal-1175.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1175.png,Normal/masks/Normal-1175.png,True,False,299,299,L,123.03,38bde7c435b50ee67ffce86df9487e48
Normal-1176.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1176.png,Normal/images/Normal-1176.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1176.png,Normal/masks/Normal-1176.png,True,False,299,299,L,130.25,6d2b0e34641667f11bc0efa58ddaa9f3
Normal-1177.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1177.png,Normal/images/Normal-1177.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1177.png,Normal/masks/Normal-1177.png,True,False,299,299,L,118.3,a0082231be034eae588d308d5c8d27ee
Normal-1178.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1178.png,Normal/images/Normal-1178.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1178.png,Normal/masks/Normal-1178.png,True,False,299,299,L,89.07,405e1d6ea16ad0c2cda38ea583e6885a
Normal-1179.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1179.png,Normal/images/Normal-1179.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1179.png,Normal/masks/Normal-1179.png,True,False,299,299,L,110.01,7b802452081b7dd93d5f87dd1f1443e9
Normal-118.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-118.png,Normal/images/Normal-118.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-118.png,Normal/masks/Normal-118.png,True,False,299,299,L,98.56,ed53b9e4d3ed0e21fcedb72fe2d64b8b
Normal-1180.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1180.png,Normal/images/Normal-1180.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1180.png,Normal/masks/Normal-1180.png,True,False,299,299,L,113.69,0f74438516504aaabb6a5cf12e1da536
Normal-1181.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1181.png,Normal/images/Normal-1181.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1181.png,Normal/masks/Normal-1181.png,True,False,299,299,L,117.09,6ef5f6ccbd06ffb8f957d5218d8ff8f2
Normal-1182.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1182.png,Normal/images/Normal-1182.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1182.png,Normal/masks/Normal-1182.png,True,False,299,299,L,124.38,1d5ef5d82d9705117e2a5f1850daea86
Normal-1183.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1183.png,Normal/images/Normal-1183.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1183.png,Normal/masks/Normal-1183.png,True,False,299,299,L,128.73,ee7ccfdce708b5e67393e51492fbf2a0
Normal-1184.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1184.png,Normal/images/Normal-1184.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1184.png,Normal/masks/Normal-1184.png,True,False,299,299,L,123.04,d17c3246b0ed79f33a39d3a20615d828
Normal-1185.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1185.png,Normal/images/Normal-1185.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1185.png,Normal/masks/Normal-1185.png,True,False,299,299,L,126.75,7d21937e074657eb8bb5d6d171789d27
Normal-1186.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1186.png,Normal/images/Normal-1186.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1186.png,Normal/masks/Normal-1186.png,True,False,299,299,L,151.59,e5af6e132e13093deebbe5ba6d2a9038
Normal-1187.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1187.png,Normal/images/Normal-1187.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1187.png,Normal/masks/Normal-1187.png,True,False,299,299,L,118.23,dd8fbf2af0961694992d38fa28c12f8f
Normal-1188.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1188.png,Normal/images/Normal-1188.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1188.png,Normal/masks/Normal-1188.png,True,False,299,299,L,145.45,2d72daf8c28841336f9d68d6669db9a5
Normal-1189.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1189.png,Normal/images/Normal-1189.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1189.png,Normal/masks/Normal-1189.png,True,False,299,299,L,122.05,cce0ef91f1e8a1bf9a60ae0bbfd9d339
Normal-119.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-119.png,Normal/images/Normal-119.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-119.png,Normal/masks/Normal-119.png,True,False,299,299,L,124.91,f3f1c7a7bf75322a027e6e70028f8d92
Normal-1190.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1190.png,Normal/images/Normal-1190.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1190.png,Normal/masks/Normal-1190.png,True,False,299,299,L,122.89,cd8b547d813a3a5b55bef33899fea5a5
Normal-1191.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1191.png,Normal/images/Normal-1191.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1191.png,Normal/masks/Normal-1191.png,True,False,299,299,L,153.66,dca6717a728b978543f9cf0b80c4d08e
Normal-1192.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1192.png,Normal/images/Normal-1192.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1192.png,Normal/masks/Normal-1192.png,True,False,299,299,L,124.57,203d301d1233f66b9a06de647cc34e06
Normal-1193.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1193.png,Normal/images/Normal-1193.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1193.png,Normal/masks/Normal-1193.png,True,False,299,299,L,141.45,364ca545ab1bb3faa85f994f1b2f7687
Normal-1194.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1194.png,Normal/images/Normal-1194.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1194.png,Normal/masks/Normal-1194.png,True,False,299,299,L,100.45,a1a317da0053a8098d5a8b61c12daddf
Normal-1195.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1195.png,Normal/images/Normal-1195.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1195.png,Normal/masks/Normal-1195.png,True,False,299,299,L,111.07,5b18ba32080a123a2d54735534df2677
Normal-1196.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1196.png,Normal/images/Normal-1196.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1196.png,Normal/masks/Normal-1196.png,True,False,299,299,L,150.04,1fd4e75bdcbb58fdd4b08c60efe8a476
Normal-1197.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1197.png,Normal/images/Normal-1197.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1197.png,Normal/masks/Normal-1197.png,True,False,299,299,L,128.69,3c1da6fb1a986dc5f3b3b43aec814c05
Normal-1198.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1198.png,Normal/images/Normal-1198.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1198.png,Normal/masks/Normal-1198.png,True,False,299,299,L,123.97,47ca5e2406222624c780b0458d21d16a
Normal-1199.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1199.png,Normal/images/Normal-1199.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1199.png,Normal/masks/Normal-1199.png,True,False,299,299,L,117.69,6feda817e1f42ebb45736cdabee4b71d
Normal-12.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-12.png,Normal/images/Normal-12.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-12.png,Normal/masks/Normal-12.png,True,False,299,299,L,121.82,15bf509e8e506e1b0861522e8692f57d
Normal-120.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-120.png,Normal/images/Normal-120.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-120.png,Normal/masks/Normal-120.png,True,False,299,299,L,134.64,fd072fdcdc3049ebbc9fdbee390b22c3
Normal-1200.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1200.png,Normal/images/Normal-1200.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1200.png,Normal/masks/Normal-1200.png,True,False,299,299,L,110.02,2c1b7def1241ec7712a5afaf5e622486
Normal-1201.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1201.png,Normal/images/Normal-1201.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1201.png,Normal/masks/Normal-1201.png,True,False,299,299,L,113.66,4048ac3409160bc16b4b0514b6a32ac0
Normal-1202.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1202.png,Normal/images/Normal-1202.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1202.png,Normal/masks/Normal-1202.png,True,False,299,299,L,135.57,6a7f4697768a2c275a2dc51500213d2c
Normal-1203.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1203.png,Normal/images/Normal-1203.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1203.png,Normal/masks/Normal-1203.png,True,False,299,299,L,133.75,36626672bcbcacbd038a2090e7189cda
Normal-1204.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1204.png,Normal/images/Normal-1204.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1204.png,Normal/masks/Normal-1204.png,True,False,299,299,L,112.57,a47a2dac677dc418217fc33ec7587aed
Normal-1205.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1205.png,Normal/images/Normal-1205.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1205.png,Normal/masks/Normal-1205.png,True,False,299,299,L,139.49,e0672ef63a2d5e6396e5f060e31690ff
Normal-1206.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1206.png,Normal/images/Normal-1206.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1206.png,Normal/masks/Normal-1206.png,True,False,299,299,L,133.93,8b5c0ee2b081bbbd1426c84b8c3d2349
Normal-1207.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1207.png,Normal/images/Normal-1207.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1207.png,Normal/masks/Normal-1207.png,True,False,299,299,L,118.61,0fa86f516cb9ebdb9a369aeeb1701012
Normal-1208.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1208.png,Normal/images/Normal-1208.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1208.png,Normal/masks/Normal-1208.png,True,False,299,299,L,118.62,4a509f6e77129b11073344d1434408ea
Normal-1209.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1209.png,Normal/images/Normal-1209.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1209.png,Normal/masks/Normal-1209.png,True,False,299,299,L,131.43,edeb6ee9579e180c7636def3956279d9
Normal-121.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-121.png,Normal/images/Normal-121.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-121.png,Normal/masks/Normal-121.png,True,False,299,299,L,122.87,718580bccbf9361dca003f020238774e
Normal-1210.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1210.png,Normal/images/Normal-1210.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1210.png,Normal/masks/Normal-1210.png,True,False,299,299,L,127.0,0b41668ca7cd57ffe2382f644bb958eb
Normal-1211.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1211.png,Normal/images/Normal-1211.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1211.png,Normal/masks/Normal-1211.png,True,False,299,299,L,106.2,62fab1dcc964428b6d3906e793df5a4f
Normal-1212.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1212.png,Normal/images/Normal-1212.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1212.png,Normal/masks/Normal-1212.png,True,False,299,299,L,124.42,aa426f7de64b02b243a80be923de3ec2
Normal-1213.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1213.png,Normal/images/Normal-1213.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1213.png,Normal/masks/Normal-1213.png,True,False,299,299,L,128.45,5acec4890d69350f4a9e734ea7da2881
Normal-1214.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1214.png,Normal/images/Normal-1214.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1214.png,Normal/masks/Normal-1214.png,True,False,299,299,L,129.29,93a7da7b17054ee40626d05b7c5902bd
Normal-1215.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1215.png,Normal/images/Normal-1215.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1215.png,Normal/masks/Normal-1215.png,True,False,299,299,L,105.04,794afcf4f1d7e55532f6258c946ff44f
Normal-1216.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1216.png,Normal/images/Normal-1216.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1216.png,Normal/masks/Normal-1216.png,True,False,299,299,L,132.93,a45a87524a39254cf550ce2eb69354a3
Normal-1217.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1217.png,Normal/images/Normal-1217.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1217.png,Normal/masks/Normal-1217.png,True,False,299,299,L,132.91,c4a110ac5c2e8a3916ddc19bd0bf59fb
Normal-1218.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1218.png,Normal/images/Normal-1218.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1218.png,Normal/masks/Normal-1218.png,True,False,299,299,L,114.71,e6a7ae7d8c18a463de49563a42d3b369
Normal-1219.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1219.png,Normal/images/Normal-1219.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1219.png,Normal/masks/Normal-1219.png,True,False,299,299,L,122.36,bd93f3f8e1f71249217ccc8b57fcbf51
Normal-122.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-122.png,Normal/images/Normal-122.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-122.png,Normal/masks/Normal-122.png,True,False,299,299,L,137.94,78ee66d6a1ccb5ffa77284b0e294764e
Normal-1220.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1220.png,Normal/images/Normal-1220.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1220.png,Normal/masks/Normal-1220.png,True,False,299,299,L,127.76,57c7a03b04c4d835b119ea143362805a
Normal-1221.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1221.png,Normal/images/Normal-1221.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1221.png,Normal/masks/Normal-1221.png,True,False,299,299,L,113.88,ae17bfe279a99c5c0970d2bf9abaa1a9
Normal-1222.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1222.png,Normal/images/Normal-1222.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1222.png,Normal/masks/Normal-1222.png,True,False,299,299,L,124.87,bc9fd10c54d77b022516248b2c85f044
Normal-1223.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1223.png,Normal/images/Normal-1223.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1223.png,Normal/masks/Normal-1223.png,True,False,299,299,L,129.57,8c81b26d5d4e0141ad62ebccd83dd0fe
Normal-1224.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1224.png,Normal/images/Normal-1224.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1224.png,Normal/masks/Normal-1224.png,True,False,299,299,L,120.04,e23199cf792d550fb5ec7f57e23cc3ff
Normal-1225.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1225.png,Normal/images/Normal-1225.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1225.png,Normal/masks/Normal-1225.png,True,False,299,299,L,124.27,d2697d9c08ce87a30accaedaffe8c14a
Normal-1226.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1226.png,Normal/images/Normal-1226.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1226.png,Normal/masks/Normal-1226.png,True,False,299,299,L,116.78,dd6d2501b588f7569fc5c27801d7c804
Normal-1227.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1227.png,Normal/images/Normal-1227.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1227.png,Normal/masks/Normal-1227.png,True,False,299,299,L,147.36,e6eadd9bc2d3fd3251561046b0499c00
Normal-1228.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1228.png,Normal/images/Normal-1228.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1228.png,Normal/masks/Normal-1228.png,True,False,299,299,L,109.64,65f977390fcc5999b16d28b1d531a783
Normal-1229.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1229.png,Normal/images/Normal-1229.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1229.png,Normal/masks/Normal-1229.png,True,False,299,299,L,134.23,1e8b07a3f3d3f3a8a9110c8b50b49ac6
Normal-123.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-123.png,Normal/images/Normal-123.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-123.png,Normal/masks/Normal-123.png,True,False,299,299,L,109.44,171d5fc4d02db631a01bce6fb7c5a349
Normal-1230.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1230.png,Normal/images/Normal-1230.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1230.png,Normal/masks/Normal-1230.png,True,False,299,299,L,129.29,14d991f0dc90790bc9ffcc6c258c4253
Normal-1231.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1231.png,Normal/images/Normal-1231.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1231.png,Normal/masks/Normal-1231.png,True,False,299,299,L,121.47,eced66f5a4286376b0d9a5f941037be3
Normal-1232.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1232.png,Normal/images/Normal-1232.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1232.png,Normal/masks/Normal-1232.png,True,False,299,299,L,139.41,9cdf6eb13ac7b1cd2f13157bc124ce88
Normal-1233.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1233.png,Normal/images/Normal-1233.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1233.png,Normal/masks/Normal-1233.png,True,False,299,299,L,114.66,1f05e45b470f9b2e365c056732126c76
Normal-1234.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1234.png,Normal/images/Normal-1234.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1234.png,Normal/masks/Normal-1234.png,True,False,299,299,L,142.11,4d5367587e13b4a5d394bdc116a8f406
Normal-1235.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1235.png,Normal/images/Normal-1235.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1235.png,Normal/masks/Normal-1235.png,True,False,299,299,L,146.35,539e0d9f845e28075efa3c568fa74ed7
Normal-1236.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1236.png,Normal/images/Normal-1236.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1236.png,Normal/masks/Normal-1236.png,True,False,299,299,L,130.33,ac3d452549e23a9b1e2a004f5eda122e
Normal-1237.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1237.png,Normal/images/Normal-1237.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1237.png,Normal/masks/Normal-1237.png,True,False,299,299,L,146.92,9034e8a68c3858b6402cfb6b62b1d46e
Normal-1238.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1238.png,Normal/images/Normal-1238.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1238.png,Normal/masks/Normal-1238.png,True,False,299,299,L,130.09,4ffa75ad70d157cf4e0a5f76cea53c18
Normal-1239.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1239.png,Normal/images/Normal-1239.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1239.png,Normal/masks/Normal-1239.png,True,False,299,299,L,115.14,1f312d57e5da3f3bd5271622339e9fec
Normal-124.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-124.png,Normal/images/Normal-124.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-124.png,Normal/masks/Normal-124.png,True,False,299,299,L,111.03,deddcf387ae22e4ed8544ba6a4369e41
Normal-1240.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1240.png,Normal/images/Normal-1240.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1240.png,Normal/masks/Normal-1240.png,True,False,299,299,L,129.62,09c85d4879019c946f111f5ff0406628
Normal-1241.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1241.png,Normal/images/Normal-1241.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1241.png,Normal/masks/Normal-1241.png,True,False,299,299,L,138.59,55aafacba184f2fb6e99b3ed36ee722c
Normal-1242.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1242.png,Normal/images/Normal-1242.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1242.png,Normal/masks/Normal-1242.png,True,False,299,299,L,131.42,15b63459d05d31cb152a354353058cae
Normal-1243.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1243.png,Normal/images/Normal-1243.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1243.png,Normal/masks/Normal-1243.png,True,False,299,299,L,136.54,9f68320ed05bb4f4c1a0bdfa729ee5ff
Normal-1244.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1244.png,Normal/images/Normal-1244.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1244.png,Normal/masks/Normal-1244.png,True,False,299,299,L,108.95,d9dd614868a63a51d99d52a01972f77a
Normal-1245.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1245.png,Normal/images/Normal-1245.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1245.png,Normal/masks/Normal-1245.png,True,False,299,299,L,126.78,ac35894e58d3650063968f3e1bce0077
Normal-1246.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1246.png,Normal/images/Normal-1246.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1246.png,Normal/masks/Normal-1246.png,True,False,299,299,L,127.48,2b25b83244089c7e4fe388780404c5a4
Normal-1247.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1247.png,Normal/images/Normal-1247.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1247.png,Normal/masks/Normal-1247.png,True,False,299,299,L,127.4,6ae831ecf36c6b4d89067df2503bbe6c
Normal-1248.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1248.png,Normal/images/Normal-1248.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1248.png,Normal/masks/Normal-1248.png,True,False,299,299,L,107.33,6fb6662d7de64feac7ec21a4da64d79e
Normal-1249.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1249.png,Normal/images/Normal-1249.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1249.png,Normal/masks/Normal-1249.png,True,False,299,299,L,117.48,9d264af3aa9f2969917f9676ddbbfb57
Normal-125.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-125.png,Normal/images/Normal-125.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-125.png,Normal/masks/Normal-125.png,True,False,299,299,L,157.47,079765b42969a0462bd1e04f19acf69e
Normal-1250.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1250.png,Normal/images/Normal-1250.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1250.png,Normal/masks/Normal-1250.png,True,False,299,299,L,124.74,ef31cf73c8593cfe64bdca5f6e386a5f
Normal-1251.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1251.png,Normal/images/Normal-1251.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1251.png,Normal/masks/Normal-1251.png,True,False,299,299,L,123.14,f6d91f2acce14ed595b05a99fabe0397
Normal-1252.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1252.png,Normal/images/Normal-1252.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1252.png,Normal/masks/Normal-1252.png,True,False,299,299,L,119.08,3bc599d38e364106a3611cb3e3244832
Normal-1253.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1253.png,Normal/images/Normal-1253.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1253.png,Normal/masks/Normal-1253.png,True,False,299,299,L,112.67,120169411ee0587a6dded5000277c3bf
Normal-1254.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1254.png,Normal/images/Normal-1254.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1254.png,Normal/masks/Normal-1254.png,True,False,299,299,L,114.88,cb7d237482f3438c0a9f5c5e16dbe95a
Normal-1255.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1255.png,Normal/images/Normal-1255.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1255.png,Normal/masks/Normal-1255.png,True,False,299,299,L,111.04,5a052ed57d2dab50f1f947203d85bf2c
Normal-1256.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1256.png,Normal/images/Normal-1256.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1256.png,Normal/masks/Normal-1256.png,True,False,299,299,L,142.0,80eb0e03d4a35ec1097f1a83a6e3e00d
Normal-1257.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1257.png,Normal/images/Normal-1257.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1257.png,Normal/masks/Normal-1257.png,True,False,299,299,L,129.62,253284cde5ea14409c432980f90f0fde
Normal-1258.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1258.png,Normal/images/Normal-1258.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1258.png,Normal/masks/Normal-1258.png,True,False,299,299,L,137.17,ddae1e465cfce3b11fceaf850ab4f3ac
Normal-1259.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1259.png,Normal/images/Normal-1259.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1259.png,Normal/masks/Normal-1259.png,True,False,299,299,L,125.61,d4137de727367444e82793b8106e22b6
Normal-126.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-126.png,Normal/images/Normal-126.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-126.png,Normal/masks/Normal-126.png,True,False,299,299,L,114.47,4cc384639297203f09790b28fad29085
Normal-1260.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1260.png,Normal/images/Normal-1260.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1260.png,Normal/masks/Normal-1260.png,True,False,299,299,L,110.27,4738177c7cf229fb5317a4e5ce641127
Normal-1261.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1261.png,Normal/images/Normal-1261.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1261.png,Normal/masks/Normal-1261.png,True,False,299,299,L,115.36,056eabf737d07c0a547424e69ece3040
Normal-1262.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1262.png,Normal/images/Normal-1262.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1262.png,Normal/masks/Normal-1262.png,True,False,299,299,L,130.21,390710430dba67638ec13a29ae9e4417
Normal-1263.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1263.png,Normal/images/Normal-1263.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1263.png,Normal/masks/Normal-1263.png,True,False,299,299,L,122.06,b7df077c16ec5f191e891ab97faceda8
Normal-1264.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1264.png,Normal/images/Normal-1264.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1264.png,Normal/masks/Normal-1264.png,True,False,299,299,L,107.13,64424af3d7a608595e6380bf982aba85
Normal-1265.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1265.png,Normal/images/Normal-1265.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1265.png,Normal/masks/Normal-1265.png,True,False,299,299,L,112.89,780e2894b558ad50af6d6339d602b6c5
Normal-1266.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1266.png,Normal/images/Normal-1266.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1266.png,Normal/masks/Normal-1266.png,True,False,299,299,L,109.95,a41964969fd133791afda07d71a2efe2
Normal-1267.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1267.png,Normal/images/Normal-1267.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1267.png,Normal/masks/Normal-1267.png,True,False,299,299,L,122.36,12fdb658ec15608a66ff049557de9b72
Normal-1268.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1268.png,Normal/images/Normal-1268.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1268.png,Normal/masks/Normal-1268.png,True,False,299,299,L,143.03,27417b2e3001eec70d889b6deb9ce0ed
Normal-1269.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1269.png,Normal/images/Normal-1269.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1269.png,Normal/masks/Normal-1269.png,True,False,299,299,L,128.93,320fb8201c6f4495783c6b03484bcd43
Normal-127.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-127.png,Normal/images/Normal-127.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-127.png,Normal/masks/Normal-127.png,True,False,299,299,L,83.79,28a7873528dc1c24f99560db052e2ff1
Normal-1270.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1270.png,Normal/images/Normal-1270.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1270.png,Normal/masks/Normal-1270.png,True,False,299,299,L,131.28,5c9f68ad3c3478be6bccdd9b1cd1b032
Normal-1271.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1271.png,Normal/images/Normal-1271.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1271.png,Normal/masks/Normal-1271.png,True,False,299,299,L,123.12,046fa0d23292f30bc87dfd025beed006
Normal-1272.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1272.png,Normal/images/Normal-1272.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1272.png,Normal/masks/Normal-1272.png,True,False,299,299,L,118.54,00eca591cb97166f1d4df34b7020eb01
Normal-1273.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1273.png,Normal/images/Normal-1273.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1273.png,Normal/masks/Normal-1273.png,True,False,299,299,L,111.9,e27769236d75f1e78910644e649d8093
Normal-1274.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1274.png,Normal/images/Normal-1274.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1274.png,Normal/masks/Normal-1274.png,True,False,299,299,L,117.37,f9d85f25a353073eaba8398cd9309d7a
Normal-1275.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1275.png,Normal/images/Normal-1275.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1275.png,Normal/masks/Normal-1275.png,True,False,299,299,L,123.59,e4670693efa52434a1a2f02134810f86
Normal-1276.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1276.png,Normal/images/Normal-1276.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1276.png,Normal/masks/Normal-1276.png,True,False,299,299,L,117.93,1fd97998c421da1d309b4e6769f8e37b
Normal-1277.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1277.png,Normal/images/Normal-1277.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1277.png,Normal/masks/Normal-1277.png,True,False,299,299,L,112.42,e07f1227fbae083050e41ed3e7d21096
Normal-1278.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1278.png,Normal/images/Normal-1278.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1278.png,Normal/masks/Normal-1278.png,True,False,299,299,L,116.64,6b54ccbf1b72543316ef7f398a18e1a9
Normal-1279.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1279.png,Normal/images/Normal-1279.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1279.png,Normal/masks/Normal-1279.png,True,False,299,299,L,116.62,2b616a8e5406c36765147873aa4cbaaa
Normal-128.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-128.png,Normal/images/Normal-128.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-128.png,Normal/masks/Normal-128.png,True,False,299,299,L,114.36,d9088ff8b51f731cdc68d843cc743ca8
Normal-1280.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1280.png,Normal/images/Normal-1280.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1280.png,Normal/masks/Normal-1280.png,True,False,299,299,L,132.44,f9de1f14953a4b1326db2551382c93a2
Normal-1281.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1281.png,Normal/images/Normal-1281.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1281.png,Normal/masks/Normal-1281.png,True,False,299,299,L,118.03,7b3bd065c26a82d42fba41f3b6da2f5b
Normal-1282.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1282.png,Normal/images/Normal-1282.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1282.png,Normal/masks/Normal-1282.png,True,False,299,299,L,119.44,d91421eb3c40b3327ecfc204b5752df1
Normal-1283.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1283.png,Normal/images/Normal-1283.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1283.png,Normal/masks/Normal-1283.png,True,False,299,299,L,99.84,2465e4d39c8ee5feddc677df82179ba1
Normal-1284.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1284.png,Normal/images/Normal-1284.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1284.png,Normal/masks/Normal-1284.png,True,False,299,299,L,109.85,e284f304a1b2cfae94d0f1537d406e1d
Normal-1285.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1285.png,Normal/images/Normal-1285.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1285.png,Normal/masks/Normal-1285.png,True,False,299,299,L,126.84,bc37b7009cf26f85ec64243b729d337e
Normal-1286.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1286.png,Normal/images/Normal-1286.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1286.png,Normal/masks/Normal-1286.png,True,False,299,299,L,108.89,66e5bca580484b0ae6aecc99d589804f
Normal-1287.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1287.png,Normal/images/Normal-1287.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1287.png,Normal/masks/Normal-1287.png,True,False,299,299,L,131.38,67c4d57bb895876793259dd58443f32e
Normal-1288.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1288.png,Normal/images/Normal-1288.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1288.png,Normal/masks/Normal-1288.png,True,False,299,299,L,143.05,3e1fccd0e95e5dd775243b4432512d5f
Normal-1289.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1289.png,Normal/images/Normal-1289.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1289.png,Normal/masks/Normal-1289.png,True,False,299,299,L,112.37,b362388f94163dcb233ac6dc6205754e
Normal-129.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-129.png,Normal/images/Normal-129.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-129.png,Normal/masks/Normal-129.png,True,False,299,299,L,121.26,731e100bb31ec8f1dd5486dabe74e4e7
Normal-1290.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1290.png,Normal/images/Normal-1290.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1290.png,Normal/masks/Normal-1290.png,True,False,299,299,L,120.15,d2ccd1869f33331c1fb0828dad18e97d
Normal-1291.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1291.png,Normal/images/Normal-1291.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1291.png,Normal/masks/Normal-1291.png,True,False,299,299,L,147.4,b04452c2419002234155307ccbd44c0f
Normal-1292.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1292.png,Normal/images/Normal-1292.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1292.png,Normal/masks/Normal-1292.png,True,False,299,299,L,123.24,7f557981d86042f9fa61d5d2d201386f
Normal-1293.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1293.png,Normal/images/Normal-1293.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1293.png,Normal/masks/Normal-1293.png,True,False,299,299,L,147.13,5e9e8d4d056caf63ad52e1462e4f1add
Normal-1294.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1294.png,Normal/images/Normal-1294.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1294.png,Normal/masks/Normal-1294.png,True,False,299,299,L,118.24,aab0b4c93fcc4bee7791a64800fedfbe
Normal-1295.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1295.png,Normal/images/Normal-1295.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1295.png,Normal/masks/Normal-1295.png,True,False,299,299,L,125.99,510941bb0c4251bd5ae4cff0f1082ef7
Normal-1296.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1296.png,Normal/images/Normal-1296.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1296.png,Normal/masks/Normal-1296.png,True,False,299,299,L,130.66,85e756c773c3867ae61a7087c9020a5d
Normal-1297.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1297.png,Normal/images/Normal-1297.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1297.png,Normal/masks/Normal-1297.png,True,False,299,299,L,147.2,ed898a4f47dba4ae1d0a4bcf15b77d44
Normal-1298.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1298.png,Normal/images/Normal-1298.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1298.png,Normal/masks/Normal-1298.png,True,False,299,299,L,149.08,d2caa2a04252a9fc32694ac7f778d449
Normal-1299.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1299.png,Normal/images/Normal-1299.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1299.png,Normal/masks/Normal-1299.png,True,False,299,299,L,130.5,5fafdc8f4e337f404c293ad5914f0c58
Normal-13.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-13.png,Normal/images/Normal-13.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-13.png,Normal/masks/Normal-13.png,True,False,299,299,L,121.01,60ea02999ae1c45aabc020273c7195c4
Normal-130.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-130.png,Normal/images/Normal-130.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-130.png,Normal/masks/Normal-130.png,True,False,299,299,L,115.65,78c7b64a27de0f2c4d935d3ab6e3241a
Normal-1300.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1300.png,Normal/images/Normal-1300.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1300.png,Normal/masks/Normal-1300.png,True,False,299,299,L,132.47,fd36a70c0a7ea0bf82d6d17b0b02b47e
Normal-1301.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1301.png,Normal/images/Normal-1301.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1301.png,Normal/masks/Normal-1301.png,True,False,299,299,L,111.5,0a7cc305dcec3604aa6ea5f3541d6d8a
Normal-1302.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1302.png,Normal/images/Normal-1302.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1302.png,Normal/masks/Normal-1302.png,True,False,299,299,L,113.89,0c4166e35f807bea684db4b137e155a6
Normal-1303.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1303.png,Normal/images/Normal-1303.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1303.png,Normal/masks/Normal-1303.png,True,False,299,299,L,104.09,2d8f5bacdf7c2a4c70862814f45d6fe6
Normal-1304.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1304.png,Normal/images/Normal-1304.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1304.png,Normal/masks/Normal-1304.png,True,False,299,299,L,99.71,2c8d31955127b1b036a0384f1f9d0644
Normal-1305.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1305.png,Normal/images/Normal-1305.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1305.png,Normal/masks/Normal-1305.png,True,False,299,299,L,122.11,2dbf3b2c1b93620eba5340bf84257ae9
Normal-1306.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1306.png,Normal/images/Normal-1306.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1306.png,Normal/masks/Normal-1306.png,True,False,299,299,L,130.34,2b8273e8f5a4e224b833e231b907d97c
Normal-1307.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1307.png,Normal/images/Normal-1307.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1307.png,Normal/masks/Normal-1307.png,True,False,299,299,L,119.14,c67d6004f370a42d87466bb5495f3fd1
Normal-1308.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1308.png,Normal/images/Normal-1308.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1308.png,Normal/masks/Normal-1308.png,True,False,299,299,L,135.99,fb116aa682b20fce5cc9ecfcd2eb8f77
Normal-1309.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1309.png,Normal/images/Normal-1309.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1309.png,Normal/masks/Normal-1309.png,True,False,299,299,L,144.4,738fb4071b477662384f30ecb03127aa
Normal-131.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-131.png,Normal/images/Normal-131.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-131.png,Normal/masks/Normal-131.png,True,False,299,299,L,124.55,6882fd9e94a55fe0b00b4b18e90a9cc6
Normal-1310.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1310.png,Normal/images/Normal-1310.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1310.png,Normal/masks/Normal-1310.png,True,False,299,299,L,126.2,9ea465333260dc6d15372d889655c0b7
Normal-1311.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1311.png,Normal/images/Normal-1311.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1311.png,Normal/masks/Normal-1311.png,True,False,299,299,L,112.06,05fbef43205e4e08927e922326d28609
Normal-1312.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1312.png,Normal/images/Normal-1312.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1312.png,Normal/masks/Normal-1312.png,True,False,299,299,L,115.13,770e0c87624a8b62c639e89767893cf3
Normal-1313.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1313.png,Normal/images/Normal-1313.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1313.png,Normal/masks/Normal-1313.png,True,False,299,299,L,139.67,2e11547e67af9a2f777f107e99a81d96
Normal-1314.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1314.png,Normal/images/Normal-1314.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1314.png,Normal/masks/Normal-1314.png,True,False,299,299,L,119.83,1f3fa456a18fc3eb875e72ca022627cb
Normal-1315.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1315.png,Normal/images/Normal-1315.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1315.png,Normal/masks/Normal-1315.png,True,False,299,299,L,130.62,543df5983d17f9a8679e6f0355a9c9c0
Normal-1316.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1316.png,Normal/images/Normal-1316.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1316.png,Normal/masks/Normal-1316.png,True,False,299,299,L,135.75,90b94b3d35a7af70887cf0abdc5e133a
Normal-1317.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1317.png,Normal/images/Normal-1317.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1317.png,Normal/masks/Normal-1317.png,True,False,299,299,L,159.27,266a18281bdfefaa5cee2782f6a98533
Normal-1318.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1318.png,Normal/images/Normal-1318.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1318.png,Normal/masks/Normal-1318.png,True,False,299,299,L,145.49,6d81bd730fafc656a1a2f5bba011ca18
Normal-1319.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1319.png,Normal/images/Normal-1319.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1319.png,Normal/masks/Normal-1319.png,True,False,299,299,L,113.89,6e202d7bc2b39243b67417d3d6eda358
Normal-132.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-132.png,Normal/images/Normal-132.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-132.png,Normal/masks/Normal-132.png,True,False,299,299,L,132.44,d6c27935e856fcf91336661252015d71
Normal-1320.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1320.png,Normal/images/Normal-1320.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1320.png,Normal/masks/Normal-1320.png,True,False,299,299,L,119.66,bf91fa18aca79fc4b63ac18aae4ce07f
Normal-1321.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1321.png,Normal/images/Normal-1321.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1321.png,Normal/masks/Normal-1321.png,True,False,299,299,L,131.98,b0ff3b65089cb2f0165233e373397598
Normal-1322.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1322.png,Normal/images/Normal-1322.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1322.png,Normal/masks/Normal-1322.png,True,False,299,299,L,135.04,b3c68a1b691183cf8ecda988ecde0cc3
Normal-1323.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1323.png,Normal/images/Normal-1323.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1323.png,Normal/masks/Normal-1323.png,True,False,299,299,L,102.71,cf67e676c58e8fe2186cc2b9fce3d40c
Normal-1324.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1324.png,Normal/images/Normal-1324.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1324.png,Normal/masks/Normal-1324.png,True,False,299,299,L,133.72,ab87cac98b91c6d0eb493ce2729a5bc6
Normal-1325.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1325.png,Normal/images/Normal-1325.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1325.png,Normal/masks/Normal-1325.png,True,False,299,299,L,115.06,14d285892c229d085d0222aeba509851
Normal-1326.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1326.png,Normal/images/Normal-1326.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1326.png,Normal/masks/Normal-1326.png,True,False,299,299,L,114.42,ad6c3d7c1dc83ef56bfdbe38e27a3f91
Normal-1327.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1327.png,Normal/images/Normal-1327.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1327.png,Normal/masks/Normal-1327.png,True,False,299,299,L,131.08,5f89ef6d0d4b91004bf9f2b036e5df49
Normal-1328.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1328.png,Normal/images/Normal-1328.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1328.png,Normal/masks/Normal-1328.png,True,False,299,299,L,107.33,34ce44b8c9443ac505cea70302beea89
Normal-1329.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1329.png,Normal/images/Normal-1329.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1329.png,Normal/masks/Normal-1329.png,True,False,299,299,L,125.0,5793b03bac58c1d1b50fb74a5cd4c414
Normal-133.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-133.png,Normal/images/Normal-133.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-133.png,Normal/masks/Normal-133.png,True,False,299,299,L,104.12,5003cce5c1a7f2286ee15cd3b7a8ba6c
Normal-1330.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1330.png,Normal/images/Normal-1330.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1330.png,Normal/masks/Normal-1330.png,True,False,299,299,L,128.2,c5ed6bbceca3757b0994b713cce5d5cc
Normal-1331.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1331.png,Normal/images/Normal-1331.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1331.png,Normal/masks/Normal-1331.png,True,False,299,299,L,123.85,d6ad61855a87b2d36704386f731cb381
Normal-1332.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1332.png,Normal/images/Normal-1332.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1332.png,Normal/masks/Normal-1332.png,True,False,299,299,L,127.92,9e4c20c514e3b3f18fc5ae413f812075
Normal-1333.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1333.png,Normal/images/Normal-1333.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1333.png,Normal/masks/Normal-1333.png,True,False,299,299,L,100.47,83524d054fa367b018ed2509751f80ed
Normal-1334.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1334.png,Normal/images/Normal-1334.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1334.png,Normal/masks/Normal-1334.png,True,False,299,299,L,125.51,8e47f319015a1b03fb48be8631d4a17a
Normal-1335.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1335.png,Normal/images/Normal-1335.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1335.png,Normal/masks/Normal-1335.png,True,False,299,299,L,96.13,41e1de78313dff34b52bd8ac6848502c
Normal-1336.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1336.png,Normal/images/Normal-1336.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1336.png,Normal/masks/Normal-1336.png,True,False,299,299,L,121.87,1388215b969873cc28633d0de59fccad
Normal-1337.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1337.png,Normal/images/Normal-1337.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1337.png,Normal/masks/Normal-1337.png,True,False,299,299,L,127.68,e690958360688d7441a1b95123636e6f
Normal-1338.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1338.png,Normal/images/Normal-1338.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1338.png,Normal/masks/Normal-1338.png,True,False,299,299,L,137.24,9a301a3a9ba287dca9ea260795f71580
Normal-1339.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1339.png,Normal/images/Normal-1339.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1339.png,Normal/masks/Normal-1339.png,True,False,299,299,L,120.72,658685afd61bdb6a5ee9101826d55ec8
Normal-134.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-134.png,Normal/images/Normal-134.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-134.png,Normal/masks/Normal-134.png,True,False,299,299,L,115.38,7a630b429613818dd9cd61e4c014fcc9
Normal-1340.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1340.png,Normal/images/Normal-1340.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1340.png,Normal/masks/Normal-1340.png,True,False,299,299,L,127.61,cce6e3ae0ca45273e59bb709558d6a4a
Normal-1341.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1341.png,Normal/images/Normal-1341.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1341.png,Normal/masks/Normal-1341.png,True,False,299,299,L,125.85,1c46b61b1c0a59140f92af60ded28607
Normal-1342.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1342.png,Normal/images/Normal-1342.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1342.png,Normal/masks/Normal-1342.png,True,False,299,299,L,116.65,b43edb6109d495dd1fb6493b2d02e481
Normal-1343.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1343.png,Normal/images/Normal-1343.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1343.png,Normal/masks/Normal-1343.png,True,False,299,299,L,151.28,6c043e82ff3d4effcf7a652f07f3801f
Normal-1344.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1344.png,Normal/images/Normal-1344.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1344.png,Normal/masks/Normal-1344.png,True,False,299,299,L,119.83,63b68f8d56cc2b94fd5ab65a4bea27bf
Normal-1345.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1345.png,Normal/images/Normal-1345.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1345.png,Normal/masks/Normal-1345.png,True,False,299,299,L,139.59,346efb19d21bf53bad8cd4afa908ce5a
Normal-1346.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1346.png,Normal/images/Normal-1346.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1346.png,Normal/masks/Normal-1346.png,True,False,299,299,L,92.55,2a46f3fe3595659f18d87bf886f048c8
Normal-1347.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1347.png,Normal/images/Normal-1347.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1347.png,Normal/masks/Normal-1347.png,True,False,299,299,L,121.76,905b358286142c378d378f1547fd1304
Normal-1348.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1348.png,Normal/images/Normal-1348.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1348.png,Normal/masks/Normal-1348.png,True,False,299,299,L,124.3,67812adaa9dd5eea45f3fad3d7282792
Normal-1349.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1349.png,Normal/images/Normal-1349.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1349.png,Normal/masks/Normal-1349.png,True,False,299,299,L,108.5,5181256bdc3b849dc7d684fcc2cf56c6
Normal-135.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-135.png,Normal/images/Normal-135.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-135.png,Normal/masks/Normal-135.png,True,False,299,299,L,100.11,ee2099bfb19cafcf5ce001b4ffa8c46c
Normal-1350.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1350.png,Normal/images/Normal-1350.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1350.png,Normal/masks/Normal-1350.png,True,False,299,299,L,138.87,3408de3e280198e91143a159d62d9585
Normal-1351.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1351.png,Normal/images/Normal-1351.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1351.png,Normal/masks/Normal-1351.png,True,False,299,299,L,124.53,ece06b31a5a56a20a4fa80b8f655805a
Normal-1352.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1352.png,Normal/images/Normal-1352.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1352.png,Normal/masks/Normal-1352.png,True,False,299,299,L,120.98,ad0047d943d0a8dd92aa0f58aac12c7b
Normal-1353.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1353.png,Normal/images/Normal-1353.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1353.png,Normal/masks/Normal-1353.png,True,False,299,299,L,115.87,07b753ad8ef0e1a1ab31bc89ddf572b2
Normal-1354.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1354.png,Normal/images/Normal-1354.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1354.png,Normal/masks/Normal-1354.png,True,False,299,299,L,140.5,821b5bea6fb292bff83454d9e70c4637
Normal-1355.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1355.png,Normal/images/Normal-1355.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1355.png,Normal/masks/Normal-1355.png,True,False,299,299,L,119.26,4f824f568b4e17691450149d28fa34db
Normal-1356.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1356.png,Normal/images/Normal-1356.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1356.png,Normal/masks/Normal-1356.png,True,False,299,299,L,105.99,ca672968613b1047f04b8942968d1645
Normal-1357.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1357.png,Normal/images/Normal-1357.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1357.png,Normal/masks/Normal-1357.png,True,False,299,299,L,145.96,0547446fd4d60e5ba384abd5e24dc44e
Normal-1358.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1358.png,Normal/images/Normal-1358.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1358.png,Normal/masks/Normal-1358.png,True,False,299,299,L,109.03,50d53053bfd720ec478ce9209ef2a9ba
Normal-1359.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1359.png,Normal/images/Normal-1359.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1359.png,Normal/masks/Normal-1359.png,True,False,299,299,L,117.92,f6d87ac1edbd0126bdc1d7c3d5d37985
Normal-136.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-136.png,Normal/images/Normal-136.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-136.png,Normal/masks/Normal-136.png,True,False,299,299,L,109.91,8a7ebec2f5e0cd131e335e5945341541
Normal-1360.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1360.png,Normal/images/Normal-1360.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1360.png,Normal/masks/Normal-1360.png,True,False,299,299,L,137.7,60e4e8181a0459a529d3f721644bc49a
Normal-1361.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1361.png,Normal/images/Normal-1361.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1361.png,Normal/masks/Normal-1361.png,True,False,299,299,L,122.83,bbffad94910098fe25347492dd86ce21
Normal-1362.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1362.png,Normal/images/Normal-1362.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1362.png,Normal/masks/Normal-1362.png,True,False,299,299,L,110.65,7865a3ffee6f44dbe12515f4573569cb
Normal-1363.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1363.png,Normal/images/Normal-1363.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1363.png,Normal/masks/Normal-1363.png,True,False,299,299,L,130.32,d99be2ec74d043566132b532411672ac
Normal-1364.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1364.png,Normal/images/Normal-1364.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1364.png,Normal/masks/Normal-1364.png,True,False,299,299,L,130.91,b787fb4da56e6a5a4dd6fee7dcd6bb7e
Normal-1365.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1365.png,Normal/images/Normal-1365.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1365.png,Normal/masks/Normal-1365.png,True,False,299,299,L,131.67,81e14c69cd3b6e4d4338e0d2166a9ef0
Normal-1366.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1366.png,Normal/images/Normal-1366.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1366.png,Normal/masks/Normal-1366.png,True,False,299,299,L,99.15,eed62cf13b9aa771e472ed3659581ee7
Normal-1367.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1367.png,Normal/images/Normal-1367.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1367.png,Normal/masks/Normal-1367.png,True,False,299,299,L,130.88,b05807169c875a80b320c0ebdd435ac5
Normal-1368.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1368.png,Normal/images/Normal-1368.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1368.png,Normal/masks/Normal-1368.png,True,False,299,299,L,113.61,39f81041b25d4aeef61b8fc44b8fd45f
Normal-1369.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1369.png,Normal/images/Normal-1369.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1369.png,Normal/masks/Normal-1369.png,True,False,299,299,L,118.18,bd2be0b3212bb9b4d6852d27935140ca
Normal-137.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-137.png,Normal/images/Normal-137.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-137.png,Normal/masks/Normal-137.png,True,False,299,299,L,110.45,3038651a0211bcba24b110d275f14541
Normal-1370.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1370.png,Normal/images/Normal-1370.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1370.png,Normal/masks/Normal-1370.png,True,False,299,299,L,104.23,de6dbd8311897b21f4ed6b147d09f094
Normal-1371.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1371.png,Normal/images/Normal-1371.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1371.png,Normal/masks/Normal-1371.png,True,False,299,299,L,119.8,15c7dae9a596238a76efe110824f9f27
Normal-1372.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1372.png,Normal/images/Normal-1372.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1372.png,Normal/masks/Normal-1372.png,True,False,299,299,L,109.93,2a2209071d5482aa0ed0043016f569f5
Normal-1373.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1373.png,Normal/images/Normal-1373.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1373.png,Normal/masks/Normal-1373.png,True,False,299,299,L,109.07,fbfe63b7e20b4fbb67e8c836fa46054b
Normal-1374.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1374.png,Normal/images/Normal-1374.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1374.png,Normal/masks/Normal-1374.png,True,False,299,299,L,102.85,1e823161980ac058f580b7a5d7e74db4
Normal-1375.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1375.png,Normal/images/Normal-1375.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1375.png,Normal/masks/Normal-1375.png,True,False,299,299,L,129.97,598195d6e8589e4e13a4a38142fb35fb
Normal-1376.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1376.png,Normal/images/Normal-1376.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1376.png,Normal/masks/Normal-1376.png,True,False,299,299,L,126.46,454d9c04a77aae14c430ee166455b49f
Normal-1377.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1377.png,Normal/images/Normal-1377.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1377.png,Normal/masks/Normal-1377.png,True,False,299,299,L,134.26,ed87f92dd714f96f4d3a3707a5e3ab2b
Normal-1378.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1378.png,Normal/images/Normal-1378.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1378.png,Normal/masks/Normal-1378.png,True,False,299,299,L,117.27,6f939db5cabc79305faa97e174a674e1
Normal-1379.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1379.png,Normal/images/Normal-1379.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1379.png,Normal/masks/Normal-1379.png,True,False,299,299,L,105.8,024e73b72bd930aaa4a1825e97487711
Normal-138.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-138.png,Normal/images/Normal-138.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-138.png,Normal/masks/Normal-138.png,True,False,299,299,L,92.91,2baac47918cf9135172c1b4f9f90dcab
Normal-1380.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1380.png,Normal/images/Normal-1380.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1380.png,Normal/masks/Normal-1380.png,True,False,299,299,L,111.15,f201d6663a462848a3ae53a6a307ea21
Normal-1381.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1381.png,Normal/images/Normal-1381.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1381.png,Normal/masks/Normal-1381.png,True,False,299,299,L,121.89,89fdb2875056739a4a1bdf252d9016e7
Normal-1382.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1382.png,Normal/images/Normal-1382.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1382.png,Normal/masks/Normal-1382.png,True,False,299,299,L,126.12,dec3665c770cb0d61a8636949756a5df
Normal-1383.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1383.png,Normal/images/Normal-1383.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1383.png,Normal/masks/Normal-1383.png,True,False,299,299,L,115.87,1b7ff19f5a1a95d8609545fcac505e81
Normal-1384.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1384.png,Normal/images/Normal-1384.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1384.png,Normal/masks/Normal-1384.png,True,False,299,299,L,117.55,bcab69496fec98f86981d67cb8b28fe3
Normal-1385.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1385.png,Normal/images/Normal-1385.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1385.png,Normal/masks/Normal-1385.png,True,False,299,299,L,121.24,eaada832cfb560e41ca41c6f869cfe6c
Normal-1386.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1386.png,Normal/images/Normal-1386.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1386.png,Normal/masks/Normal-1386.png,True,False,299,299,L,133.95,47919887ed0ad65ac50b7172463a636d
Normal-1387.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1387.png,Normal/images/Normal-1387.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1387.png,Normal/masks/Normal-1387.png,True,False,299,299,L,129.72,c9f5014ecbe00df249aa643828a83c72
Normal-1388.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1388.png,Normal/images/Normal-1388.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1388.png,Normal/masks/Normal-1388.png,True,False,299,299,L,131.81,f342c1cae6bbca86b8f38177e9b95ca0
Normal-1389.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1389.png,Normal/images/Normal-1389.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1389.png,Normal/masks/Normal-1389.png,True,False,299,299,L,133.99,3597adb8560b4c422b186fd890d56403
Normal-139.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-139.png,Normal/images/Normal-139.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-139.png,Normal/masks/Normal-139.png,True,False,299,299,L,120.33,602865050a08d82362ad72634708db28
Normal-1390.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1390.png,Normal/images/Normal-1390.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1390.png,Normal/masks/Normal-1390.png,True,False,299,299,L,109.1,bfa2bc10097f0bdafaba1b662bbf39fe
Normal-1391.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1391.png,Normal/images/Normal-1391.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1391.png,Normal/masks/Normal-1391.png,True,False,299,299,L,135.9,716a70e132e3295606780f9afa461161
Normal-1392.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1392.png,Normal/images/Normal-1392.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1392.png,Normal/masks/Normal-1392.png,True,False,299,299,L,121.69,b9d2b94eae57517a6381d4c0aaac5148
Normal-1393.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1393.png,Normal/images/Normal-1393.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1393.png,Normal/masks/Normal-1393.png,True,False,299,299,L,134.85,8624e9011bfa44d1bea5ebdc0a91209c
Normal-1394.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1394.png,Normal/images/Normal-1394.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1394.png,Normal/masks/Normal-1394.png,True,False,299,299,L,136.48,b68fcbbbc0cc1d1b46f719c7dfbfc0f6
Normal-1395.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1395.png,Normal/images/Normal-1395.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1395.png,Normal/masks/Normal-1395.png,True,False,299,299,L,111.78,7d8c60bb8c6adeaa591ecdaccd3dde08
Normal-1396.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1396.png,Normal/images/Normal-1396.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1396.png,Normal/masks/Normal-1396.png,True,False,299,299,L,122.02,76f00a6d9b3f14ff70b743dd59b27c6c
Normal-1397.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1397.png,Normal/images/Normal-1397.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1397.png,Normal/masks/Normal-1397.png,True,False,299,299,L,124.92,91d1038c3342d052090108ea1b3c69ee
Normal-1398.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1398.png,Normal/images/Normal-1398.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1398.png,Normal/masks/Normal-1398.png,True,False,299,299,L,142.2,6089f00af99aeff335a59a2f1b5f953c
Normal-1399.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1399.png,Normal/images/Normal-1399.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1399.png,Normal/masks/Normal-1399.png,True,False,299,299,L,128.97,fe4557741d2da6d92d84e083855bc64c
Normal-14.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-14.png,Normal/images/Normal-14.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-14.png,Normal/masks/Normal-14.png,True,False,299,299,L,157.62,00b53ecba8ec61de19096b38f526bec7
Normal-140.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-140.png,Normal/images/Normal-140.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-140.png,Normal/masks/Normal-140.png,True,False,299,299,L,119.67,248b1a7bd7bd5b1477539a63f92514f5
Normal-1400.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1400.png,Normal/images/Normal-1400.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1400.png,Normal/masks/Normal-1400.png,True,False,299,299,L,113.86,b59eb73612ca91a16a0db7f7115beb71
Normal-1401.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1401.png,Normal/images/Normal-1401.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1401.png,Normal/masks/Normal-1401.png,True,False,299,299,L,106.58,4a68c327bc27f9ab6889f1da8883419b
Normal-1402.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1402.png,Normal/images/Normal-1402.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1402.png,Normal/masks/Normal-1402.png,True,False,299,299,L,123.65,8901d328baa13a63f0236d96c7edd7d1
Normal-1403.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1403.png,Normal/images/Normal-1403.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1403.png,Normal/masks/Normal-1403.png,True,False,299,299,L,102.36,c295c127eb955c0d97fbca6007845201
Normal-1404.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1404.png,Normal/images/Normal-1404.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1404.png,Normal/masks/Normal-1404.png,True,False,299,299,L,114.33,8bdaf6da73f6f34530e731b4a98a118d
Normal-1405.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1405.png,Normal/images/Normal-1405.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1405.png,Normal/masks/Normal-1405.png,True,False,299,299,L,158.72,60da7274d915c7464404e12adad32c30
Normal-1406.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1406.png,Normal/images/Normal-1406.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1406.png,Normal/masks/Normal-1406.png,True,False,299,299,L,136.3,6fc18e58c18a094f5262a319a2744cb3
Normal-1407.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1407.png,Normal/images/Normal-1407.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1407.png,Normal/masks/Normal-1407.png,True,False,299,299,L,109.19,1abe053cb9a01dc83e0f17e857ae2a81
Normal-1408.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1408.png,Normal/images/Normal-1408.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1408.png,Normal/masks/Normal-1408.png,True,False,299,299,L,174.67,3deb61a6f6542d1005de9dbe27903cb5
Normal-1409.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1409.png,Normal/images/Normal-1409.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1409.png,Normal/masks/Normal-1409.png,True,False,299,299,L,116.93,437e0475a3550d7c860ad9dbe2951e54
Normal-141.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-141.png,Normal/images/Normal-141.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-141.png,Normal/masks/Normal-141.png,True,False,299,299,L,133.32,948b0ada6c5a0cf002a246e707fa027d
Normal-1410.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1410.png,Normal/images/Normal-1410.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1410.png,Normal/masks/Normal-1410.png,True,False,299,299,L,128.3,9ed147d49c875d5dc2e47159fd08c289
Normal-1411.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1411.png,Normal/images/Normal-1411.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1411.png,Normal/masks/Normal-1411.png,True,False,299,299,L,100.68,39f4b5a41b0805b22de1bbd85e6eb94c
Normal-1412.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1412.png,Normal/images/Normal-1412.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1412.png,Normal/masks/Normal-1412.png,True,False,299,299,L,160.56,f61a33d2fe37835cbdaf054b1e941c0c
Normal-1413.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1413.png,Normal/images/Normal-1413.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1413.png,Normal/masks/Normal-1413.png,True,False,299,299,L,166.36,e1c44b96679658be019fbdb9a72a89fb
Normal-1414.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1414.png,Normal/images/Normal-1414.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1414.png,Normal/masks/Normal-1414.png,True,False,299,299,L,113.38,18c684e7ad1872971252f50b1b3f0a5e
Normal-1415.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1415.png,Normal/images/Normal-1415.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1415.png,Normal/masks/Normal-1415.png,True,False,299,299,L,154.81,f3cc36c940b0f44abf383d5e4b4d700f
Normal-1416.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1416.png,Normal/images/Normal-1416.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1416.png,Normal/masks/Normal-1416.png,True,False,299,299,L,123.02,cc082bdbc1b9982fb6f63949ff8451ec
Normal-1417.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1417.png,Normal/images/Normal-1417.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1417.png,Normal/masks/Normal-1417.png,True,False,299,299,L,134.59,2bb71485d49f3dda2a324511a27d268b
Normal-1418.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1418.png,Normal/images/Normal-1418.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1418.png,Normal/masks/Normal-1418.png,True,False,299,299,L,180.51,366007488d2172094154b9dcf3b58ebb
Normal-1419.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1419.png,Normal/images/Normal-1419.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1419.png,Normal/masks/Normal-1419.png,True,False,299,299,L,103.52,6930276a6bfcd742b496e0a434e80230
Normal-142.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-142.png,Normal/images/Normal-142.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-142.png,Normal/masks/Normal-142.png,True,False,299,299,L,117.16,d01526de5fb16827f464c8df2de4ce34
Normal-1420.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1420.png,Normal/images/Normal-1420.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1420.png,Normal/masks/Normal-1420.png,True,False,299,299,L,111.58,0df27646907f68ac219e424f23d11d98
Normal-1421.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1421.png,Normal/images/Normal-1421.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1421.png,Normal/masks/Normal-1421.png,True,False,299,299,L,164.31,d2086709b86fcae6b4f02f52116ea7aa
Normal-1422.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1422.png,Normal/images/Normal-1422.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1422.png,Normal/masks/Normal-1422.png,True,False,299,299,L,156.11,a96ac071855da18a627d09723ef172e4
Normal-1423.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1423.png,Normal/images/Normal-1423.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1423.png,Normal/masks/Normal-1423.png,True,False,299,299,L,109.7,27d4ccb672bebc1f31856977817bbb9d
Normal-1424.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1424.png,Normal/images/Normal-1424.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1424.png,Normal/masks/Normal-1424.png,True,False,299,299,L,162.36,b36a24548fcb568a4805c65cbd20dc4f
Normal-1425.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1425.png,Normal/images/Normal-1425.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1425.png,Normal/masks/Normal-1425.png,True,False,299,299,L,165.74,0b9e9718ac8ff2f63002727c49c167bf
Normal-1426.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1426.png,Normal/images/Normal-1426.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1426.png,Normal/masks/Normal-1426.png,True,False,299,299,L,111.24,6ba634f478e10691a6c0da5c300dde25
Normal-1427.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1427.png,Normal/images/Normal-1427.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1427.png,Normal/masks/Normal-1427.png,True,False,299,299,L,128.96,35d55fe5c384c08cb443c6e120132e31
Normal-1428.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1428.png,Normal/images/Normal-1428.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1428.png,Normal/masks/Normal-1428.png,True,False,299,299,L,165.11,a5bec2c7ba695644590ae532ba90e45f
Normal-1429.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1429.png,Normal/images/Normal-1429.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1429.png,Normal/masks/Normal-1429.png,True,False,299,299,L,151.38,a0864653495cb00ed543d33dfa91c55e
Normal-143.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-143.png,Normal/images/Normal-143.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-143.png,Normal/masks/Normal-143.png,True,False,299,299,L,110.86,c769c87618a97731cc91eee77b620105
Normal-1430.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1430.png,Normal/images/Normal-1430.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1430.png,Normal/masks/Normal-1430.png,True,False,299,299,L,129.77,8448448b637c45e69ff5289e5632aa50
Normal-1431.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1431.png,Normal/images/Normal-1431.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1431.png,Normal/masks/Normal-1431.png,True,False,299,299,L,157.32,3313eaa37511936fe35d9f51b6e27c6a
Normal-1432.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1432.png,Normal/images/Normal-1432.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1432.png,Normal/masks/Normal-1432.png,True,False,299,299,L,133.36,69e840c7234daaf6bf8b094562243571
Normal-1433.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1433.png,Normal/images/Normal-1433.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1433.png,Normal/masks/Normal-1433.png,True,False,299,299,L,133.23,f57312bf513bd284169e1de84933009f
Normal-1434.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1434.png,Normal/images/Normal-1434.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1434.png,Normal/masks/Normal-1434.png,True,False,299,299,L,170.41,be0d2dc16ae27660ada20a4293083472
Normal-1435.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1435.png,Normal/images/Normal-1435.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1435.png,Normal/masks/Normal-1435.png,True,False,299,299,L,167.06,d532c95b7662d73ae375a2264ebb1bfd
Normal-1436.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1436.png,Normal/images/Normal-1436.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1436.png,Normal/masks/Normal-1436.png,True,False,299,299,L,98.21,46d746879a47ad0bfee568bd07ba5677
Normal-1437.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1437.png,Normal/images/Normal-1437.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1437.png,Normal/masks/Normal-1437.png,True,False,299,299,L,121.71,4ad907f17279847bfb8de04b4b2c9abe
Normal-1438.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1438.png,Normal/images/Normal-1438.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1438.png,Normal/masks/Normal-1438.png,True,False,299,299,L,115.72,d2b7d1948b838bed0935e408c071fe0f
Normal-1439.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1439.png,Normal/images/Normal-1439.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1439.png,Normal/masks/Normal-1439.png,True,False,299,299,L,112.37,2191c32659f80fb644032fa9a1eff732
Normal-144.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-144.png,Normal/images/Normal-144.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-144.png,Normal/masks/Normal-144.png,True,False,299,299,L,125.94,d4cd26b8f4b8a9fa0ede4c0b7185a521
Normal-1440.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1440.png,Normal/images/Normal-1440.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1440.png,Normal/masks/Normal-1440.png,True,False,299,299,L,138.53,5037fb389f0ce881be7ef4870b1df99a
Normal-1441.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1441.png,Normal/images/Normal-1441.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1441.png,Normal/masks/Normal-1441.png,True,False,299,299,L,157.71,0bbf2e54bbdc14025a723d95c2a0eb96
Normal-1442.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1442.png,Normal/images/Normal-1442.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1442.png,Normal/masks/Normal-1442.png,True,False,299,299,L,141.36,16fa12bf04e7e22752389d1cf58fd6d3
Normal-1443.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1443.png,Normal/images/Normal-1443.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1443.png,Normal/masks/Normal-1443.png,True,False,299,299,L,129.58,31b98111a1b4d970645e39aeca1b2e7d
Normal-1444.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1444.png,Normal/images/Normal-1444.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1444.png,Normal/masks/Normal-1444.png,True,False,299,299,L,172.48,62f2f433d01e6c3b59d743cd94028b21
Normal-1445.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1445.png,Normal/images/Normal-1445.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1445.png,Normal/masks/Normal-1445.png,True,False,299,299,L,124.36,3e1554cff527ba8f08f19690b8eeec7f
Normal-1446.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1446.png,Normal/images/Normal-1446.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1446.png,Normal/masks/Normal-1446.png,True,False,299,299,L,129.14,9fe292e3d4ba31c621bdf2b4cc66c0ea
Normal-1447.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1447.png,Normal/images/Normal-1447.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1447.png,Normal/masks/Normal-1447.png,True,False,299,299,L,181.12,abff3d1e7ea6ccb3f3a1ec68556bfce5
Normal-1448.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1448.png,Normal/images/Normal-1448.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1448.png,Normal/masks/Normal-1448.png,True,False,299,299,L,113.09,0c0423f710553395d35ff9dee996d26c
Normal-1449.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1449.png,Normal/images/Normal-1449.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1449.png,Normal/masks/Normal-1449.png,True,False,299,299,L,116.0,7076a7accb723b15acbe2685c4762022
Normal-145.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-145.png,Normal/images/Normal-145.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-145.png,Normal/masks/Normal-145.png,True,False,299,299,L,87.32,f4c3ee27b2dc8114f178ab549b1506b7
Normal-1450.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1450.png,Normal/images/Normal-1450.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1450.png,Normal/masks/Normal-1450.png,True,False,299,299,L,111.57,41ff015993b134af22ead668c33a64dc
Normal-1451.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1451.png,Normal/images/Normal-1451.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1451.png,Normal/masks/Normal-1451.png,True,False,299,299,L,158.5,039264ea476ab86feef730b3728fcd89
Normal-1452.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1452.png,Normal/images/Normal-1452.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1452.png,Normal/masks/Normal-1452.png,True,False,299,299,L,166.93,441d61e043abaeef8ffa0b8445089f8a
Normal-1453.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1453.png,Normal/images/Normal-1453.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1453.png,Normal/masks/Normal-1453.png,True,False,299,299,L,121.5,e8aeade66ab4d38b630d29cc9b262591
Normal-1454.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1454.png,Normal/images/Normal-1454.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1454.png,Normal/masks/Normal-1454.png,True,False,299,299,L,160.17,d3365c219f023d6f749a4a8dece30827
Normal-1455.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1455.png,Normal/images/Normal-1455.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1455.png,Normal/masks/Normal-1455.png,True,False,299,299,L,77.62,b8c68466df234e12469581a254142ced
Normal-1456.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1456.png,Normal/images/Normal-1456.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1456.png,Normal/masks/Normal-1456.png,True,False,299,299,L,108.65,6bdd324838cb90503cdc5fdd223bb748
Normal-1457.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1457.png,Normal/images/Normal-1457.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1457.png,Normal/masks/Normal-1457.png,True,False,299,299,L,149.5,e59d1b73e811cb0f9af33c4f39ea8f8a
Normal-1458.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1458.png,Normal/images/Normal-1458.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1458.png,Normal/masks/Normal-1458.png,True,False,299,299,L,169.4,c1a9335286abeaf6ed8612fdb32c37c6
Normal-1459.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1459.png,Normal/images/Normal-1459.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1459.png,Normal/masks/Normal-1459.png,True,False,299,299,L,156.61,d59645867c2f650f81c1877113c5957e
Normal-146.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-146.png,Normal/images/Normal-146.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-146.png,Normal/masks/Normal-146.png,True,False,299,299,L,118.6,3d8ec2d8e09102c0259e50846299d024
Normal-1460.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1460.png,Normal/images/Normal-1460.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1460.png,Normal/masks/Normal-1460.png,True,False,299,299,L,106.7,0b0a26eeb96c30d60bb5724f73b9149e
Normal-1461.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1461.png,Normal/images/Normal-1461.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1461.png,Normal/masks/Normal-1461.png,True,False,299,299,L,166.31,2a562a7e96b76cca54bab21d99059e37
Normal-1462.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1462.png,Normal/images/Normal-1462.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1462.png,Normal/masks/Normal-1462.png,True,False,299,299,L,134.79,ccf3671e8a79b91ef7de27e22bd8d7af
Normal-1463.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1463.png,Normal/images/Normal-1463.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1463.png,Normal/masks/Normal-1463.png,True,False,299,299,L,191.07,cdd7030c97e4a1dcd24b67e0e85061a5
Normal-1464.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1464.png,Normal/images/Normal-1464.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1464.png,Normal/masks/Normal-1464.png,True,False,299,299,L,116.96,49eebd52ef998e07e54586413ce0e822
Normal-1465.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1465.png,Normal/images/Normal-1465.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1465.png,Normal/masks/Normal-1465.png,True,False,299,299,L,127.84,7258cdfaab495e406238a896e982bf53
Normal-1466.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1466.png,Normal/images/Normal-1466.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1466.png,Normal/masks/Normal-1466.png,True,False,299,299,L,84.27,cb45c3f7cc484b7e2cb3cbca4ea11faa
Normal-1467.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1467.png,Normal/images/Normal-1467.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1467.png,Normal/masks/Normal-1467.png,True,False,299,299,L,141.2,82bc8f6a2a7ef5300248c5bb25e72abc
Normal-1468.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1468.png,Normal/images/Normal-1468.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1468.png,Normal/masks/Normal-1468.png,True,False,299,299,L,75.74,2ce0472bc36913f87af541b52c2b985a
Normal-1469.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1469.png,Normal/images/Normal-1469.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1469.png,Normal/masks/Normal-1469.png,True,False,299,299,L,121.45,9876377e0b2f65302071309557bded0a
Normal-147.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-147.png,Normal/images/Normal-147.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-147.png,Normal/masks/Normal-147.png,True,False,299,299,L,125.92,ace5335664545c506ded32adfb617614
Normal-1470.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1470.png,Normal/images/Normal-1470.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1470.png,Normal/masks/Normal-1470.png,True,False,299,299,L,142.44,e1e8303d294c2f7ff52e384f3da27273
Normal-1471.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1471.png,Normal/images/Normal-1471.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1471.png,Normal/masks/Normal-1471.png,True,False,299,299,L,122.67,3806b760ec51435deeb0aad0a1604143
Normal-1472.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1472.png,Normal/images/Normal-1472.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1472.png,Normal/masks/Normal-1472.png,True,False,299,299,L,111.6,ef594e82c3a19ab9e7bee987b5385351
Normal-1473.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1473.png,Normal/images/Normal-1473.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1473.png,Normal/masks/Normal-1473.png,True,False,299,299,L,130.93,94eb6d3fc2834cf50676ccec411a3f1b
Normal-1474.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1474.png,Normal/images/Normal-1474.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1474.png,Normal/masks/Normal-1474.png,True,False,299,299,L,116.84,d1f7122daf85b3bc7be68b8300ca2ed4
Normal-1475.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1475.png,Normal/images/Normal-1475.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1475.png,Normal/masks/Normal-1475.png,True,False,299,299,L,100.58,1e8cf6805140c873b3092baee1f9c561
Normal-1476.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1476.png,Normal/images/Normal-1476.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1476.png,Normal/masks/Normal-1476.png,True,False,299,299,L,120.46,33b7a2c7ea5759549352d855530d2ba2
Normal-1477.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1477.png,Normal/images/Normal-1477.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1477.png,Normal/masks/Normal-1477.png,True,False,299,299,L,95.49,891c55a4af25f7101115ca54c532123a
Normal-1478.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1478.png,Normal/images/Normal-1478.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1478.png,Normal/masks/Normal-1478.png,True,False,299,299,L,119.24,2a49d3620daaffb3f48b930e9524d543
Normal-1479.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1479.png,Normal/images/Normal-1479.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1479.png,Normal/masks/Normal-1479.png,True,False,299,299,L,127.15,bfbb33c4732afaf0db2c176b11b84dd1
Normal-148.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-148.png,Normal/images/Normal-148.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-148.png,Normal/masks/Normal-148.png,True,False,299,299,L,126.46,7954a375a3ddd8f4f8512fd14aa84e02
Normal-1480.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1480.png,Normal/images/Normal-1480.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1480.png,Normal/masks/Normal-1480.png,True,False,299,299,L,150.07,a1a7ec1b5e09cd667821df77283be29b
Normal-1481.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1481.png,Normal/images/Normal-1481.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1481.png,Normal/masks/Normal-1481.png,True,False,299,299,L,112.35,3cc8b808436ead0b0f1d64d9f6fcd23b
Normal-1482.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1482.png,Normal/images/Normal-1482.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1482.png,Normal/masks/Normal-1482.png,True,False,299,299,L,111.59,8b8be82184bc0307afab39afc095628d
Normal-1483.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1483.png,Normal/images/Normal-1483.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1483.png,Normal/masks/Normal-1483.png,True,False,299,299,L,146.68,e6535a8b340cacc3a33898559af81e2d
Normal-1484.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1484.png,Normal/images/Normal-1484.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1484.png,Normal/masks/Normal-1484.png,True,False,299,299,L,142.07,76b64a3674ba5c9de44ec2758506ed03
Normal-1485.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1485.png,Normal/images/Normal-1485.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1485.png,Normal/masks/Normal-1485.png,True,False,299,299,L,130.05,b1dc2050c26d7fdde511951248b123b6
Normal-1486.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1486.png,Normal/images/Normal-1486.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1486.png,Normal/masks/Normal-1486.png,True,False,299,299,L,139.49,645b4c14a50e2550b3b0c74d9e2b2840
Normal-1487.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1487.png,Normal/images/Normal-1487.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1487.png,Normal/masks/Normal-1487.png,True,False,299,299,L,109.27,2311ea6c96b7c4ba5246794ebebdb3c7
Normal-1488.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1488.png,Normal/images/Normal-1488.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1488.png,Normal/masks/Normal-1488.png,True,False,299,299,L,118.84,3a679e7362a76b6399595cada9b6da87
Normal-1489.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1489.png,Normal/images/Normal-1489.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1489.png,Normal/masks/Normal-1489.png,True,False,299,299,L,116.93,7c42a0c4e8157ef88d61f374bc8fe553
Normal-149.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-149.png,Normal/images/Normal-149.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-149.png,Normal/masks/Normal-149.png,True,False,299,299,L,121.74,4543549880a9de13ec71699579828768
Normal-1490.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1490.png,Normal/images/Normal-1490.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1490.png,Normal/masks/Normal-1490.png,True,False,299,299,L,164.27,7487d323f6bd758918f0e6a5ff8de030
Normal-1491.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1491.png,Normal/images/Normal-1491.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1491.png,Normal/masks/Normal-1491.png,True,False,299,299,L,162.21,44ff40228dd79dc6dae3b22f45b3fbe1
Normal-1492.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1492.png,Normal/images/Normal-1492.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1492.png,Normal/masks/Normal-1492.png,True,False,299,299,L,162.24,c478f4a98e7617983080043ad37004f5
Normal-1493.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1493.png,Normal/images/Normal-1493.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1493.png,Normal/masks/Normal-1493.png,True,False,299,299,L,107.21,222c705130880c9926fb1db1f8e3e092
Normal-1494.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1494.png,Normal/images/Normal-1494.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1494.png,Normal/masks/Normal-1494.png,True,False,299,299,L,125.1,f1f3ed458def02f992daec7163a48a84
Normal-1495.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1495.png,Normal/images/Normal-1495.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1495.png,Normal/masks/Normal-1495.png,True,False,299,299,L,130.1,8ef94bdb26bcd4377951ab448f06d2c4
Normal-1496.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1496.png,Normal/images/Normal-1496.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1496.png,Normal/masks/Normal-1496.png,True,False,299,299,L,123.74,0d72310afc5d753813223768178e3ec4
Normal-1497.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1497.png,Normal/images/Normal-1497.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1497.png,Normal/masks/Normal-1497.png,True,False,299,299,L,142.4,5f95fba7b72fd6238b1f7811d8361422
Normal-1498.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1498.png,Normal/images/Normal-1498.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1498.png,Normal/masks/Normal-1498.png,True,False,299,299,L,131.35,d5244feb04d08020245c52e34348ddc1
Normal-1499.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1499.png,Normal/images/Normal-1499.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1499.png,Normal/masks/Normal-1499.png,True,False,299,299,L,129.87,27f2c40579623e90270a480a93503013
Normal-15.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-15.png,Normal/images/Normal-15.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-15.png,Normal/masks/Normal-15.png,True,False,299,299,L,120.97,2e4aa79b094e5aec5c43512cdccf399c
Normal-150.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-150.png,Normal/images/Normal-150.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-150.png,Normal/masks/Normal-150.png,True,False,299,299,L,106.09,74b489ffc423b6f5d602054f15af7268
Normal-1500.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1500.png,Normal/images/Normal-1500.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1500.png,Normal/masks/Normal-1500.png,True,False,299,299,L,73.43,458a9b4a36e2a44e2c6fc3375be9e05a
Normal-1501.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1501.png,Normal/images/Normal-1501.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1501.png,Normal/masks/Normal-1501.png,True,False,299,299,L,118.65,385fa89c899909d5cdcedfbc117c9d57
Normal-1502.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1502.png,Normal/images/Normal-1502.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1502.png,Normal/masks/Normal-1502.png,True,False,299,299,L,147.66,c6104db3ab7548e101747cc623a82b94
Normal-1503.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1503.png,Normal/images/Normal-1503.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1503.png,Normal/masks/Normal-1503.png,True,False,299,299,L,118.69,0ed8f0dc641df913369d07085875ce79
Normal-1504.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1504.png,Normal/images/Normal-1504.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1504.png,Normal/masks/Normal-1504.png,True,False,299,299,L,132.52,52dd408c4e3939f3e6212548b4704159
Normal-1505.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1505.png,Normal/images/Normal-1505.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1505.png,Normal/masks/Normal-1505.png,True,False,299,299,L,178.52,59c0bbd30c1300bb6e6efa13f124c94b
Normal-1506.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1506.png,Normal/images/Normal-1506.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1506.png,Normal/masks/Normal-1506.png,True,False,299,299,L,136.44,5b2b9c9f8ce777e5a44020e8a457dc37
Normal-1507.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1507.png,Normal/images/Normal-1507.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1507.png,Normal/masks/Normal-1507.png,True,False,299,299,L,120.56,886f274ada542af77524d805887ee3e6
Normal-1508.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1508.png,Normal/images/Normal-1508.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1508.png,Normal/masks/Normal-1508.png,True,False,299,299,L,137.0,7572c1b083104de5ece1d40b5a856aba
Normal-1509.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1509.png,Normal/images/Normal-1509.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1509.png,Normal/masks/Normal-1509.png,True,False,299,299,L,163.45,9af34f41429e779872b5c9f28ad2c81d
Normal-151.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-151.png,Normal/images/Normal-151.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-151.png,Normal/masks/Normal-151.png,True,False,299,299,L,102.72,617689b9eab722e6b716e1c5139e1615
Normal-1510.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1510.png,Normal/images/Normal-1510.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1510.png,Normal/masks/Normal-1510.png,True,False,299,299,L,170.19,4836901841cd300b91207e4d2fc106f5
Normal-1511.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1511.png,Normal/images/Normal-1511.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1511.png,Normal/masks/Normal-1511.png,True,False,299,299,L,123.3,bfe16b3dc48c46f1c76a3c3e96ec9a93
Normal-1512.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1512.png,Normal/images/Normal-1512.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1512.png,Normal/masks/Normal-1512.png,True,False,299,299,L,129.47,5fdeb5910c44b48c3c6b6b1636727508
Normal-1513.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1513.png,Normal/images/Normal-1513.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1513.png,Normal/masks/Normal-1513.png,True,False,299,299,L,121.48,2e1bf6305ad2830078540475a4f2da46
Normal-1514.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1514.png,Normal/images/Normal-1514.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1514.png,Normal/masks/Normal-1514.png,True,False,299,299,L,123.78,bb795e43fd61e75a27da5aa545a947b2
Normal-1515.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1515.png,Normal/images/Normal-1515.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1515.png,Normal/masks/Normal-1515.png,True,False,299,299,L,114.01,8f953fb1d3d638532ef62b43355f3191
Normal-1516.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1516.png,Normal/images/Normal-1516.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1516.png,Normal/masks/Normal-1516.png,True,False,299,299,L,168.08,af15be6e61bae9a456ab1f2615c5ec89
Normal-1517.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1517.png,Normal/images/Normal-1517.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1517.png,Normal/masks/Normal-1517.png,True,False,299,299,L,145.81,0b6bc8bfd9b5ddd5e328f60d0b4b1dbb
Normal-1518.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1518.png,Normal/images/Normal-1518.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1518.png,Normal/masks/Normal-1518.png,True,False,299,299,L,109.97,2070278261ac7daac3c4367cd77276c5
Normal-1519.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1519.png,Normal/images/Normal-1519.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1519.png,Normal/masks/Normal-1519.png,True,False,299,299,L,157.27,4440173a08d9b0aade7276c47f72e595
Normal-152.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-152.png,Normal/images/Normal-152.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-152.png,Normal/masks/Normal-152.png,True,False,299,299,L,115.58,62e736a3d03ef52a1bd62684c1a3dad8
Normal-1520.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1520.png,Normal/images/Normal-1520.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1520.png,Normal/masks/Normal-1520.png,True,False,299,299,L,121.65,3343f895d523f5cd1707aa1e78196603
Normal-1521.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1521.png,Normal/images/Normal-1521.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1521.png,Normal/masks/Normal-1521.png,True,False,299,299,L,133.95,ca3d9c7473250e30afc9fd290f5cd822
Normal-1522.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1522.png,Normal/images/Normal-1522.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1522.png,Normal/masks/Normal-1522.png,True,False,299,299,L,129.32,99878ce7673bb282c9e30fc785de8590
Normal-1523.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1523.png,Normal/images/Normal-1523.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1523.png,Normal/masks/Normal-1523.png,True,False,299,299,L,131.81,b6b22d6addcc3d609f90177fdde2a1ba
Normal-1524.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1524.png,Normal/images/Normal-1524.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1524.png,Normal/masks/Normal-1524.png,True,False,299,299,L,138.71,219ca6c51be09a5edbf32d7fa80b3d7e
Normal-1525.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1525.png,Normal/images/Normal-1525.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1525.png,Normal/masks/Normal-1525.png,True,False,299,299,L,127.2,1cce7d3d25a6096f2c239ab20b159194
Normal-1526.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1526.png,Normal/images/Normal-1526.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1526.png,Normal/masks/Normal-1526.png,True,False,299,299,L,157.61,2743b805869a4a7dc158a98eb443ad6a
Normal-1527.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1527.png,Normal/images/Normal-1527.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1527.png,Normal/masks/Normal-1527.png,True,False,299,299,L,126.16,82c673bbca6c7fcbc304f84f4a2a8f52
Normal-1528.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1528.png,Normal/images/Normal-1528.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1528.png,Normal/masks/Normal-1528.png,True,False,299,299,L,151.42,f69ca3219db8dfde706744c002f446e5
Normal-1529.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1529.png,Normal/images/Normal-1529.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1529.png,Normal/masks/Normal-1529.png,True,False,299,299,L,104.43,1117414d8268c93738888d4de8c11f34
Normal-153.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-153.png,Normal/images/Normal-153.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-153.png,Normal/masks/Normal-153.png,True,False,299,299,L,133.5,282703e08342637daab8378565d9051a
Normal-1530.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1530.png,Normal/images/Normal-1530.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1530.png,Normal/masks/Normal-1530.png,True,False,299,299,L,131.91,1acb786f5a5c9bbc4cc007d860a95fe5
Normal-1531.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1531.png,Normal/images/Normal-1531.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1531.png,Normal/masks/Normal-1531.png,True,False,299,299,L,103.75,1ee3b6eb0030d3fb2ddacbf00a5a0c67
Normal-1532.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1532.png,Normal/images/Normal-1532.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1532.png,Normal/masks/Normal-1532.png,True,False,299,299,L,153.53,f4feb710053e0f3a7f9d263248f1fa75
Normal-1533.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1533.png,Normal/images/Normal-1533.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1533.png,Normal/masks/Normal-1533.png,True,False,299,299,L,101.33,7d81c8b8247227c16909af59a1c02d06
Normal-1534.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1534.png,Normal/images/Normal-1534.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1534.png,Normal/masks/Normal-1534.png,True,False,299,299,L,120.69,5a3bc53e4bf145d661942e0ec2bdece8
Normal-1535.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1535.png,Normal/images/Normal-1535.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1535.png,Normal/masks/Normal-1535.png,True,False,299,299,L,108.58,3ad53457fc89c9ef52a1789d0d1e9256
Normal-1536.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1536.png,Normal/images/Normal-1536.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1536.png,Normal/masks/Normal-1536.png,True,False,299,299,L,172.12,fed9f1ab9895556f44b71446db2f4848
Normal-1537.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1537.png,Normal/images/Normal-1537.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1537.png,Normal/masks/Normal-1537.png,True,False,299,299,L,83.65,eeffb128605bda965883e34203b9f923
Normal-1538.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1538.png,Normal/images/Normal-1538.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1538.png,Normal/masks/Normal-1538.png,True,False,299,299,L,155.61,8814d82b18bcd4911035c08700f34186
Normal-1539.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1539.png,Normal/images/Normal-1539.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1539.png,Normal/masks/Normal-1539.png,True,False,299,299,L,125.64,a87c78b4d253779336bb4cc4dcd74617
Normal-154.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-154.png,Normal/images/Normal-154.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-154.png,Normal/masks/Normal-154.png,True,False,299,299,L,135.06,380ba59379664fc8a6fd8f0c6743ae53
Normal-1540.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1540.png,Normal/images/Normal-1540.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1540.png,Normal/masks/Normal-1540.png,True,False,299,299,L,118.03,b13442ca1b470aff356abd4afd29f688
Normal-1541.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1541.png,Normal/images/Normal-1541.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1541.png,Normal/masks/Normal-1541.png,True,False,299,299,L,115.75,7aae24c932488827122697eac974fbcf
Normal-1542.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1542.png,Normal/images/Normal-1542.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1542.png,Normal/masks/Normal-1542.png,True,False,299,299,L,137.94,b8031a2725c408087dd7152af73ee899
Normal-1543.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1543.png,Normal/images/Normal-1543.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1543.png,Normal/masks/Normal-1543.png,True,False,299,299,L,113.95,1136fc95e4c97088367b41a87ea29053
Normal-1544.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1544.png,Normal/images/Normal-1544.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1544.png,Normal/masks/Normal-1544.png,True,False,299,299,L,104.02,14b99518e5c3b28a5fc1dde8fb9c8aff
Normal-1545.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1545.png,Normal/images/Normal-1545.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1545.png,Normal/masks/Normal-1545.png,True,False,299,299,L,158.92,32f19a7bde95d1e3f7d657170212c76d
Normal-1546.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1546.png,Normal/images/Normal-1546.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1546.png,Normal/masks/Normal-1546.png,True,False,299,299,L,123.68,384d2d881049f8ef64783641b79b56a0
Normal-1547.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1547.png,Normal/images/Normal-1547.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1547.png,Normal/masks/Normal-1547.png,True,False,299,299,L,95.17,8ef318bea826c991ed2a52f53c8702c1
Normal-1548.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1548.png,Normal/images/Normal-1548.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1548.png,Normal/masks/Normal-1548.png,True,False,299,299,L,179.13,335bab5d8c95a8731c621504e705b519
Normal-1549.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1549.png,Normal/images/Normal-1549.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1549.png,Normal/masks/Normal-1549.png,True,False,299,299,L,105.5,2cfb9b33c1a244b0606c40caf7580f0e
Normal-155.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-155.png,Normal/images/Normal-155.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-155.png,Normal/masks/Normal-155.png,True,False,299,299,L,98.52,bec48e6bb2f4d4941dc0854bbbc11da1
Normal-1550.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1550.png,Normal/images/Normal-1550.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1550.png,Normal/masks/Normal-1550.png,True,False,299,299,L,144.75,23f8a6c0387a8fd2b434ead0854f9e2d
Normal-1551.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1551.png,Normal/images/Normal-1551.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1551.png,Normal/masks/Normal-1551.png,True,False,299,299,L,116.49,5cd79f3dc86ce68901236081e380c262
Normal-1552.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1552.png,Normal/images/Normal-1552.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1552.png,Normal/masks/Normal-1552.png,True,False,299,299,L,112.42,8c501ab67f9ef5cad6d87fd33301c880
Normal-1553.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1553.png,Normal/images/Normal-1553.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1553.png,Normal/masks/Normal-1553.png,True,False,299,299,L,136.88,47db29ace278ab39ca7ae34b7f24d826
Normal-1554.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1554.png,Normal/images/Normal-1554.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1554.png,Normal/masks/Normal-1554.png,True,False,299,299,L,137.35,f7e866ea3ec25987854b0a830167f128
Normal-1555.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1555.png,Normal/images/Normal-1555.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1555.png,Normal/masks/Normal-1555.png,True,False,299,299,L,113.18,20b6bc4fbb11a5eada1c37c16970fc06
Normal-1556.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1556.png,Normal/images/Normal-1556.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1556.png,Normal/masks/Normal-1556.png,True,False,299,299,L,107.11,1b66d4e04bcc772181955b2ab9d6b2d3
Normal-1557.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1557.png,Normal/images/Normal-1557.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1557.png,Normal/masks/Normal-1557.png,True,False,299,299,L,126.14,53f86d7300ba13cb0c78e5fd80c72de2
Normal-1558.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1558.png,Normal/images/Normal-1558.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1558.png,Normal/masks/Normal-1558.png,True,False,299,299,L,89.96,00258c96ad32148e3312dd1088718418
Normal-1559.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1559.png,Normal/images/Normal-1559.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1559.png,Normal/masks/Normal-1559.png,True,False,299,299,L,113.15,345593a2d12f1750d8289a1fe380f9bf
Normal-156.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-156.png,Normal/images/Normal-156.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-156.png,Normal/masks/Normal-156.png,True,False,299,299,L,90.3,7d63c8711b394cb22f58fa8116648052
Normal-1560.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1560.png,Normal/images/Normal-1560.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1560.png,Normal/masks/Normal-1560.png,True,False,299,299,L,118.24,68dd431ba10d4948f42a4ad516a9b6dd
Normal-1561.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1561.png,Normal/images/Normal-1561.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1561.png,Normal/masks/Normal-1561.png,True,False,299,299,L,173.29,46b8d20f546d37adc2b2b0534d3063b0
Normal-1562.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1562.png,Normal/images/Normal-1562.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1562.png,Normal/masks/Normal-1562.png,True,False,299,299,L,145.87,c368f31430c14b0668db1c6aa7dcec68
Normal-1563.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1563.png,Normal/images/Normal-1563.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1563.png,Normal/masks/Normal-1563.png,True,False,299,299,L,158.11,bc78ed32ecb9d68a3edd0b2afad09877
Normal-1564.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1564.png,Normal/images/Normal-1564.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1564.png,Normal/masks/Normal-1564.png,True,False,299,299,L,159.53,d243b7e553742f6bca286de093d1c41d
Normal-1565.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1565.png,Normal/images/Normal-1565.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1565.png,Normal/masks/Normal-1565.png,True,False,299,299,L,159.46,c168736e41feac8abf9b8e9631d4b166
Normal-1566.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1566.png,Normal/images/Normal-1566.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1566.png,Normal/masks/Normal-1566.png,True,False,299,299,L,186.43,5e3aa1d0d7ae373388598c5786c1f337
Normal-1567.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1567.png,Normal/images/Normal-1567.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1567.png,Normal/masks/Normal-1567.png,True,False,299,299,L,119.93,7144c64877003f1b5110670ef607714a
Normal-1568.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1568.png,Normal/images/Normal-1568.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1568.png,Normal/masks/Normal-1568.png,True,False,299,299,L,106.36,9ad441c57e39a9ee0a5911f173d23638
Normal-1569.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1569.png,Normal/images/Normal-1569.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1569.png,Normal/masks/Normal-1569.png,True,False,299,299,L,160.24,653bafc3075a2c635572fc0a26c1c124
Normal-157.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-157.png,Normal/images/Normal-157.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-157.png,Normal/masks/Normal-157.png,True,False,299,299,L,113.86,fa7211dc2039c96dfca443b02e05f3f5
Normal-1570.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1570.png,Normal/images/Normal-1570.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1570.png,Normal/masks/Normal-1570.png,True,False,299,299,L,109.14,f939df6d5914d2dcba7841256455b370
Normal-1571.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1571.png,Normal/images/Normal-1571.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1571.png,Normal/masks/Normal-1571.png,True,False,299,299,L,116.48,6681d086b65860d652213dd54071e70f
Normal-1572.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1572.png,Normal/images/Normal-1572.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1572.png,Normal/masks/Normal-1572.png,True,False,299,299,L,89.93,5778aef33f699d365acddd04a79c8079
Normal-1573.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1573.png,Normal/images/Normal-1573.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1573.png,Normal/masks/Normal-1573.png,True,False,299,299,L,126.94,f449a7f3d0f5bd8a2037d80ac0311b80
Normal-1574.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1574.png,Normal/images/Normal-1574.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1574.png,Normal/masks/Normal-1574.png,True,False,299,299,L,137.45,4671179b71de087d877ae7a710c1961b
Normal-1575.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1575.png,Normal/images/Normal-1575.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1575.png,Normal/masks/Normal-1575.png,True,False,299,299,L,102.74,d7572362a588ec50bcac68efa985f0ac
Normal-1576.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1576.png,Normal/images/Normal-1576.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1576.png,Normal/masks/Normal-1576.png,True,False,299,299,L,153.78,0dfcffa4d14812e0b88833d4040b0590
Normal-1577.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1577.png,Normal/images/Normal-1577.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1577.png,Normal/masks/Normal-1577.png,True,False,299,299,L,138.48,5da1fdcbe55d4715e4d2c5b502a17c9b
Normal-1578.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1578.png,Normal/images/Normal-1578.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1578.png,Normal/masks/Normal-1578.png,True,False,299,299,L,111.83,46c31ecd7b9477fbb210ade821ddc845
Normal-1579.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1579.png,Normal/images/Normal-1579.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1579.png,Normal/masks/Normal-1579.png,True,False,299,299,L,164.34,d43a0f9e2489fb3fc7a17be85b7b8842
Normal-158.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-158.png,Normal/images/Normal-158.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-158.png,Normal/masks/Normal-158.png,True,False,299,299,L,142.75,49d7cb9ee7c40c2d78c2925456295bc6
Normal-1580.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1580.png,Normal/images/Normal-1580.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1580.png,Normal/masks/Normal-1580.png,True,False,299,299,L,130.09,d0c8925d017abc1ce1eb68fb337bbbb5
Normal-1581.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1581.png,Normal/images/Normal-1581.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1581.png,Normal/masks/Normal-1581.png,True,False,299,299,L,123.09,d2f4956e50e1468613f1ac17a19321f5
Normal-1582.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1582.png,Normal/images/Normal-1582.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1582.png,Normal/masks/Normal-1582.png,True,False,299,299,L,110.4,230a28dbf0e4fd7a48864bcb48c89e05
Normal-1583.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1583.png,Normal/images/Normal-1583.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1583.png,Normal/masks/Normal-1583.png,True,False,299,299,L,120.03,7a30c33534c03ced06c810fc4bb9d26c
Normal-1584.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1584.png,Normal/images/Normal-1584.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1584.png,Normal/masks/Normal-1584.png,True,False,299,299,L,138.11,3dd82c5c9d06afed7909c360e03f22e5
Normal-1585.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1585.png,Normal/images/Normal-1585.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1585.png,Normal/masks/Normal-1585.png,True,False,299,299,L,102.33,a1ec33d5318b3e14a9761ffbae7b2bb8
Normal-1586.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1586.png,Normal/images/Normal-1586.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1586.png,Normal/masks/Normal-1586.png,True,False,299,299,L,120.71,d2336e1fcb53f39b090fb679a8434384
Normal-1587.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1587.png,Normal/images/Normal-1587.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1587.png,Normal/masks/Normal-1587.png,True,False,299,299,L,110.33,0c0ed9a707027ea0b64920e4944f5e7c
Normal-1588.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1588.png,Normal/images/Normal-1588.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1588.png,Normal/masks/Normal-1588.png,True,False,299,299,L,117.41,0f3a6c31888dd8251a1c73f57548511a
Normal-1589.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1589.png,Normal/images/Normal-1589.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1589.png,Normal/masks/Normal-1589.png,True,False,299,299,L,157.54,703883acbf232acc1d8b04633f74d4e4
Normal-159.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-159.png,Normal/images/Normal-159.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-159.png,Normal/masks/Normal-159.png,True,False,299,299,L,147.61,58ae9b12f66f9775433865867475ab69
Normal-1590.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1590.png,Normal/images/Normal-1590.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1590.png,Normal/masks/Normal-1590.png,True,False,299,299,L,165.34,97498d85a8dd95a1e001921aaa7d7018
Normal-1591.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1591.png,Normal/images/Normal-1591.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1591.png,Normal/masks/Normal-1591.png,True,False,299,299,L,145.25,d44e3b7db1fb17e2132c7dacc44c973e
Normal-1592.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1592.png,Normal/images/Normal-1592.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1592.png,Normal/masks/Normal-1592.png,True,False,299,299,L,116.65,0ff9d7ff24646d8654efb7bcedbbee02
Normal-1593.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1593.png,Normal/images/Normal-1593.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1593.png,Normal/masks/Normal-1593.png,True,False,299,299,L,173.66,0b0ee3a5c1ea6b4a959429710b4238b8
Normal-1594.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1594.png,Normal/images/Normal-1594.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1594.png,Normal/masks/Normal-1594.png,True,False,299,299,L,109.8,b2a1c363532a2df33249c950f9202667
Normal-1595.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1595.png,Normal/images/Normal-1595.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1595.png,Normal/masks/Normal-1595.png,True,False,299,299,L,165.4,7a3ff6b3056327f796e7e288be938825
Normal-1596.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1596.png,Normal/images/Normal-1596.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1596.png,Normal/masks/Normal-1596.png,True,False,299,299,L,142.06,a5af14782c554962b4c41cee5f140a03
Normal-1597.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1597.png,Normal/images/Normal-1597.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1597.png,Normal/masks/Normal-1597.png,True,False,299,299,L,132.28,6fa1a9c91edd799d1ca8cf7fcc7d4585
Normal-1598.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1598.png,Normal/images/Normal-1598.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1598.png,Normal/masks/Normal-1598.png,True,False,299,299,L,179.83,3f8384c633e0d0ac935fddeb5f36b6bd
Normal-1599.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1599.png,Normal/images/Normal-1599.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1599.png,Normal/masks/Normal-1599.png,True,False,299,299,L,154.12,3fa3d3dce106acb4535987f68d3d2cf1
Normal-16.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-16.png,Normal/images/Normal-16.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-16.png,Normal/masks/Normal-16.png,True,False,299,299,L,143.77,a4526acb3d93ca521eca1000a2031ff1
Normal-160.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-160.png,Normal/images/Normal-160.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-160.png,Normal/masks/Normal-160.png,True,False,299,299,L,122.79,9c2747dc5037ef14f1445c9c00180c0a
Normal-1600.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1600.png,Normal/images/Normal-1600.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1600.png,Normal/masks/Normal-1600.png,True,False,299,299,L,110.82,7c4f507e0bfa556d3ee6ae06218bb3d3
Normal-1601.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1601.png,Normal/images/Normal-1601.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1601.png,Normal/masks/Normal-1601.png,True,False,299,299,L,119.31,053cab85cc058c1605b70097e60c738f
Normal-1602.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1602.png,Normal/images/Normal-1602.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1602.png,Normal/masks/Normal-1602.png,True,False,299,299,L,121.71,a6d56ecc4b6e5beb560b5e627224fd05
Normal-1603.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1603.png,Normal/images/Normal-1603.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1603.png,Normal/masks/Normal-1603.png,True,False,299,299,L,115.4,97c317cf7718d70db23c96a4b2e98e49
Normal-1604.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1604.png,Normal/images/Normal-1604.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1604.png,Normal/masks/Normal-1604.png,True,False,299,299,L,117.6,537e084873ed23846bef4a5b59e9ec3e
Normal-1605.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1605.png,Normal/images/Normal-1605.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1605.png,Normal/masks/Normal-1605.png,True,False,299,299,L,169.5,4afc2c2e857a3ed05e71313b9d06234b
Normal-1606.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1606.png,Normal/images/Normal-1606.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1606.png,Normal/masks/Normal-1606.png,True,False,299,299,L,110.45,7b03b3a2714a15828410fc7a52f1e89c
Normal-1607.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1607.png,Normal/images/Normal-1607.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1607.png,Normal/masks/Normal-1607.png,True,False,299,299,L,176.38,46b2ed69d63bd1301ef06ff559bb50e4
Normal-1608.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1608.png,Normal/images/Normal-1608.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1608.png,Normal/masks/Normal-1608.png,True,False,299,299,L,102.08,36a7858c5e594fe66b04f938b91d190a
Normal-1609.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1609.png,Normal/images/Normal-1609.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1609.png,Normal/masks/Normal-1609.png,True,False,299,299,L,103.99,be087ed8e5cb6859d1778c83f7a3dc0b
Normal-161.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-161.png,Normal/images/Normal-161.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-161.png,Normal/masks/Normal-161.png,True,False,299,299,L,130.3,b54c7e1a731841b2de0ecb4d3e21a515
Normal-1610.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1610.png,Normal/images/Normal-1610.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1610.png,Normal/masks/Normal-1610.png,True,False,299,299,L,120.2,08fcb8b0936e9eb480830dac6a6142c4
Normal-1611.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1611.png,Normal/images/Normal-1611.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1611.png,Normal/masks/Normal-1611.png,True,False,299,299,L,133.51,d9123c07b4fa006577d564338bef9172
Normal-1612.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1612.png,Normal/images/Normal-1612.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1612.png,Normal/masks/Normal-1612.png,True,False,299,299,L,92.29,3afc991029e1c8e73c043ab5e24cbd18
Normal-1613.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1613.png,Normal/images/Normal-1613.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1613.png,Normal/masks/Normal-1613.png,True,False,299,299,L,154.11,091bb0a4e40b860f972611eeaa403585
Normal-1614.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1614.png,Normal/images/Normal-1614.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1614.png,Normal/masks/Normal-1614.png,True,False,299,299,L,130.67,7ac317c5bfb10702c97e105757888d9c
Normal-1615.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1615.png,Normal/images/Normal-1615.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1615.png,Normal/masks/Normal-1615.png,True,False,299,299,L,183.24,4af70c5b9d9857d8173a935eff6277f9
Normal-1616.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1616.png,Normal/images/Normal-1616.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1616.png,Normal/masks/Normal-1616.png,True,False,299,299,L,157.54,3af51a7b6610785ec3c4f34d7846d848
Normal-1617.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1617.png,Normal/images/Normal-1617.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1617.png,Normal/masks/Normal-1617.png,True,False,299,299,L,158.97,8307daf5a0d0ca4c44d15cdf2528c7b7
Normal-1618.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1618.png,Normal/images/Normal-1618.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1618.png,Normal/masks/Normal-1618.png,True,False,299,299,L,168.32,f4d1c6a50b9f1e5950d7036926191c57
Normal-1619.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1619.png,Normal/images/Normal-1619.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1619.png,Normal/masks/Normal-1619.png,True,False,299,299,L,157.93,132ecc5ceaad344b6bdc61995865964b
Normal-162.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-162.png,Normal/images/Normal-162.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-162.png,Normal/masks/Normal-162.png,True,False,299,299,L,113.94,502971a68cd932dba986d7ef795471cd
Normal-1620.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1620.png,Normal/images/Normal-1620.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1620.png,Normal/masks/Normal-1620.png,True,False,299,299,L,125.38,5dc333069a643419159ed4079df8ce71
Normal-1621.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1621.png,Normal/images/Normal-1621.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1621.png,Normal/masks/Normal-1621.png,True,False,299,299,L,120.44,b102f1fe7da4399ebc55b5b2a3848227
Normal-1622.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1622.png,Normal/images/Normal-1622.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1622.png,Normal/masks/Normal-1622.png,True,False,299,299,L,128.77,e8ff1bec8939449924f094682cc7d75b
Normal-1623.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1623.png,Normal/images/Normal-1623.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1623.png,Normal/masks/Normal-1623.png,True,False,299,299,L,134.53,e02478668d058ea6c183ac9803e03379
Normal-1624.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1624.png,Normal/images/Normal-1624.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1624.png,Normal/masks/Normal-1624.png,True,False,299,299,L,123.84,7613f36e09d69b2664e596f6a3f1499f
Normal-1625.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1625.png,Normal/images/Normal-1625.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1625.png,Normal/masks/Normal-1625.png,True,False,299,299,L,106.21,5c354fa98a2dbb7cfbd52ecc47cb1bbf
Normal-1626.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1626.png,Normal/images/Normal-1626.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1626.png,Normal/masks/Normal-1626.png,True,False,299,299,L,131.12,df72bde0147b6422dff641b090022747
Normal-1627.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1627.png,Normal/images/Normal-1627.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1627.png,Normal/masks/Normal-1627.png,True,False,299,299,L,86.09,fe09777730d7662bd2e3d31f5a23a2e6
Normal-1628.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1628.png,Normal/images/Normal-1628.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1628.png,Normal/masks/Normal-1628.png,True,False,299,299,L,128.72,b0b8c97c242b13c9f99a235d95ade68b
Normal-1629.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1629.png,Normal/images/Normal-1629.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1629.png,Normal/masks/Normal-1629.png,True,False,299,299,L,141.43,6b57868f3d1b62424aa3e77b42810685
Normal-163.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-163.png,Normal/images/Normal-163.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-163.png,Normal/masks/Normal-163.png,True,False,299,299,L,134.0,5c7a46bbde112acce0e89d639a96bd11
Normal-1630.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1630.png,Normal/images/Normal-1630.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1630.png,Normal/masks/Normal-1630.png,True,False,299,299,L,135.54,30f1254d8e3d32c6013bc755241f046a
Normal-1631.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1631.png,Normal/images/Normal-1631.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1631.png,Normal/masks/Normal-1631.png,True,False,299,299,L,114.23,f0f31d2ae0c83ecd1edca4bb13108ab7
Normal-1632.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1632.png,Normal/images/Normal-1632.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1632.png,Normal/masks/Normal-1632.png,True,False,299,299,L,165.69,e3688dec530aaea7f99a8637e0ae8f76
Normal-1633.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1633.png,Normal/images/Normal-1633.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1633.png,Normal/masks/Normal-1633.png,True,False,299,299,L,104.58,2a512a2e95b32e3313da69832401fdaf
Normal-1634.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1634.png,Normal/images/Normal-1634.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1634.png,Normal/masks/Normal-1634.png,True,False,299,299,L,168.42,e80325c2e42fb127f39aa29b9a644fb5
Normal-1635.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1635.png,Normal/images/Normal-1635.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1635.png,Normal/masks/Normal-1635.png,True,False,299,299,L,151.28,d357d4bd6596b2b7913bd29b9ccd381d
Normal-1636.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1636.png,Normal/images/Normal-1636.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1636.png,Normal/masks/Normal-1636.png,True,False,299,299,L,178.98,54802a1d7512f39573d23b636800a2c8
Normal-1637.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1637.png,Normal/images/Normal-1637.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1637.png,Normal/masks/Normal-1637.png,True,False,299,299,L,113.26,cc567fd3d446b0345c7efee33cec0ae2
Normal-1638.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1638.png,Normal/images/Normal-1638.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1638.png,Normal/masks/Normal-1638.png,True,False,299,299,L,111.78,5c6b7157051d9d6ed3c1fe95d4db71be
Normal-1639.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1639.png,Normal/images/Normal-1639.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1639.png,Normal/masks/Normal-1639.png,True,False,299,299,L,99.38,2a689aba9b334f3227861ee778a2ba9c
Normal-164.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-164.png,Normal/images/Normal-164.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-164.png,Normal/masks/Normal-164.png,True,False,299,299,L,105.9,b3b7a3e6ec01b220a68224aabc6913ab
Normal-1640.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1640.png,Normal/images/Normal-1640.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1640.png,Normal/masks/Normal-1640.png,True,False,299,299,L,127.8,0db844bb498fccce00f4edc73e3bc66a
Normal-1641.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1641.png,Normal/images/Normal-1641.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1641.png,Normal/masks/Normal-1641.png,True,False,299,299,L,114.04,cb09bfa01a543b8de1504f1675fe9295
Normal-1642.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1642.png,Normal/images/Normal-1642.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1642.png,Normal/masks/Normal-1642.png,True,False,299,299,L,103.11,8ee22f32c3a3795aeec4e61112c07981
Normal-1643.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1643.png,Normal/images/Normal-1643.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1643.png,Normal/masks/Normal-1643.png,True,False,299,299,L,173.93,fd8035b44759571cd84522fe6514db3b
Normal-1644.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1644.png,Normal/images/Normal-1644.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1644.png,Normal/masks/Normal-1644.png,True,False,299,299,L,132.45,e5201bcae8423ac64d136af8d7507d6b
Normal-1645.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1645.png,Normal/images/Normal-1645.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1645.png,Normal/masks/Normal-1645.png,True,False,299,299,L,99.73,729e16803c22a4c5897834fc71807080
Normal-1646.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1646.png,Normal/images/Normal-1646.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1646.png,Normal/masks/Normal-1646.png,True,False,299,299,L,128.07,cc31f36884ba658c706c2fc1e5234e21
Normal-1647.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1647.png,Normal/images/Normal-1647.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1647.png,Normal/masks/Normal-1647.png,True,False,299,299,L,161.87,85f768f789b2b906074a199117e573a1
Normal-1648.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1648.png,Normal/images/Normal-1648.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1648.png,Normal/masks/Normal-1648.png,True,False,299,299,L,110.92,c5c6ed57773fb0b2dc470b12387f31c4
Normal-1649.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1649.png,Normal/images/Normal-1649.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1649.png,Normal/masks/Normal-1649.png,True,False,299,299,L,100.23,2772e65799780d9273035effaf406871
Normal-165.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-165.png,Normal/images/Normal-165.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-165.png,Normal/masks/Normal-165.png,True,False,299,299,L,139.43,7b40de8f018992bf8121818737fe4480
Normal-1650.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1650.png,Normal/images/Normal-1650.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1650.png,Normal/masks/Normal-1650.png,True,False,299,299,L,116.81,a5b29ccdf6b6ae2e02a24d5a2b20be95
Normal-1651.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1651.png,Normal/images/Normal-1651.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1651.png,Normal/masks/Normal-1651.png,True,False,299,299,L,132.88,4c1745f9dd3566a70af4482243c012b4
Normal-1652.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1652.png,Normal/images/Normal-1652.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1652.png,Normal/masks/Normal-1652.png,True,False,299,299,L,157.13,3f0288b4a1c9c57914ef689b3df5d54d
Normal-1653.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1653.png,Normal/images/Normal-1653.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1653.png,Normal/masks/Normal-1653.png,True,False,299,299,L,124.29,c246d6ade666e88b0abdcee2a4dcb4c9
Normal-1654.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1654.png,Normal/images/Normal-1654.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1654.png,Normal/masks/Normal-1654.png,True,False,299,299,L,118.82,d5444ae4fec2c72f32a3c3c2eb5dbd91
Normal-1655.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1655.png,Normal/images/Normal-1655.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1655.png,Normal/masks/Normal-1655.png,True,False,299,299,L,105.04,9478d78ec3cba0c87c3bb8d9e4920e8c
Normal-1656.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1656.png,Normal/images/Normal-1656.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1656.png,Normal/masks/Normal-1656.png,True,False,299,299,L,107.98,03982e08900e7215f73f2223b8e3c003
Normal-1657.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1657.png,Normal/images/Normal-1657.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1657.png,Normal/masks/Normal-1657.png,True,False,299,299,L,142.82,b56dd5b88e571af8660a05bcfad121b9
Normal-1658.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1658.png,Normal/images/Normal-1658.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1658.png,Normal/masks/Normal-1658.png,True,False,299,299,L,153.39,35320271957fa3000e7ac4797acf7239
Normal-1659.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1659.png,Normal/images/Normal-1659.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1659.png,Normal/masks/Normal-1659.png,True,False,299,299,L,51.8,0b40d87b75d8aacbf1541abdbc432488
Normal-166.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-166.png,Normal/images/Normal-166.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-166.png,Normal/masks/Normal-166.png,True,False,299,299,L,128.9,450376ceb8bc1098f188ff85159dc544
Normal-1660.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1660.png,Normal/images/Normal-1660.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1660.png,Normal/masks/Normal-1660.png,True,False,299,299,L,172.21,85c7bd8d4a448a82907dec746857e889
Normal-1661.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1661.png,Normal/images/Normal-1661.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1661.png,Normal/masks/Normal-1661.png,True,False,299,299,L,125.13,25ea9f177dffbd0bd475a940d1a475ad
Normal-1662.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1662.png,Normal/images/Normal-1662.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1662.png,Normal/masks/Normal-1662.png,True,False,299,299,L,143.56,9abc630e2046357c7cd581ff2b06ed78
Normal-1663.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1663.png,Normal/images/Normal-1663.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1663.png,Normal/masks/Normal-1663.png,True,False,299,299,L,165.99,4366a6a6750805657761ef9ef0dfd664
Normal-1664.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1664.png,Normal/images/Normal-1664.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1664.png,Normal/masks/Normal-1664.png,True,False,299,299,L,104.92,1a966fb25fbf0b720fdbc9600ed17e3f
Normal-1665.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1665.png,Normal/images/Normal-1665.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1665.png,Normal/masks/Normal-1665.png,True,False,299,299,L,114.46,bb19c03defd62585788452900336d3c5
Normal-1666.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1666.png,Normal/images/Normal-1666.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1666.png,Normal/masks/Normal-1666.png,True,False,299,299,L,114.08,567a1ddec4896daef06ac7ea590a75a4
Normal-1667.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1667.png,Normal/images/Normal-1667.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1667.png,Normal/masks/Normal-1667.png,True,False,299,299,L,121.65,704c8bbe4a4e47be52ce67df031ace8e
Normal-1668.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1668.png,Normal/images/Normal-1668.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1668.png,Normal/masks/Normal-1668.png,True,False,299,299,L,133.29,b4e0f299d2923bdb0dcc9acc0ae4a55f
Normal-1669.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1669.png,Normal/images/Normal-1669.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1669.png,Normal/masks/Normal-1669.png,True,False,299,299,L,133.07,1c58ec56e3f1ed0b724f1ee9b1fe47cc
Normal-167.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-167.png,Normal/images/Normal-167.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-167.png,Normal/masks/Normal-167.png,True,False,299,299,L,130.03,6b216309b68a3e4b6d225c2c6a6d6f13
Normal-1670.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1670.png,Normal/images/Normal-1670.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1670.png,Normal/masks/Normal-1670.png,True,False,299,299,L,161.56,2d627f1f1e6eac49a6e1be8efb70524a
Normal-1671.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1671.png,Normal/images/Normal-1671.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1671.png,Normal/masks/Normal-1671.png,True,False,299,299,L,112.52,a9da640fc085455f99995bfa25c487f2
Normal-1672.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1672.png,Normal/images/Normal-1672.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1672.png,Normal/masks/Normal-1672.png,True,False,299,299,L,96.33,dcf9fa05ec3c91e131afaaaf68f7e314
Normal-1673.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1673.png,Normal/images/Normal-1673.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1673.png,Normal/masks/Normal-1673.png,True,False,299,299,L,121.65,7e513b6854befad1e04b1d49a0927b87
Normal-1674.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1674.png,Normal/images/Normal-1674.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1674.png,Normal/masks/Normal-1674.png,True,False,299,299,L,155.39,3ba1f37003d84e0911b640d24c46bca1
Normal-1675.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1675.png,Normal/images/Normal-1675.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1675.png,Normal/masks/Normal-1675.png,True,False,299,299,L,132.84,4df38b245dbe06e9a634aa319ff639d3
Normal-1676.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1676.png,Normal/images/Normal-1676.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1676.png,Normal/masks/Normal-1676.png,True,False,299,299,L,124.78,338e39af9de0e42d541cdb323aa961bd
Normal-1677.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1677.png,Normal/images/Normal-1677.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1677.png,Normal/masks/Normal-1677.png,True,False,299,299,L,160.36,cf48d59b8ace182ba4920dde8fa865c5
Normal-1678.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1678.png,Normal/images/Normal-1678.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1678.png,Normal/masks/Normal-1678.png,True,False,299,299,L,109.95,1812350b48f4bcc845c4bba73e2a2d6e
Normal-1679.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1679.png,Normal/images/Normal-1679.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1679.png,Normal/masks/Normal-1679.png,True,False,299,299,L,144.02,9cc9724c14595bb8334eda18527e1ee8
Normal-168.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-168.png,Normal/images/Normal-168.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-168.png,Normal/masks/Normal-168.png,True,False,299,299,L,115.98,2e9e1b270bc6173c04363d67ff939778
Normal-1680.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1680.png,Normal/images/Normal-1680.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1680.png,Normal/masks/Normal-1680.png,True,False,299,299,L,99.31,63805127b4b3fa43bcef6bfaac87ee33
Normal-1681.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1681.png,Normal/images/Normal-1681.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1681.png,Normal/masks/Normal-1681.png,True,False,299,299,L,99.72,0d023156937c7f6840e3c217a32b82af
Normal-1682.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1682.png,Normal/images/Normal-1682.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1682.png,Normal/masks/Normal-1682.png,True,False,299,299,L,172.72,8b65bc1c98d175ef1c01544185283577
Normal-1683.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1683.png,Normal/images/Normal-1683.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1683.png,Normal/masks/Normal-1683.png,True,False,299,299,L,118.15,3954f6bcdeef4710cf57e9d6e5866b71
Normal-1684.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1684.png,Normal/images/Normal-1684.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1684.png,Normal/masks/Normal-1684.png,True,False,299,299,L,124.36,2297083c5a79252687006d1bb0691b53
Normal-1685.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1685.png,Normal/images/Normal-1685.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1685.png,Normal/masks/Normal-1685.png,True,False,299,299,L,106.93,2d3fee9d92e57c5066921d9a10ea6fca
Normal-1686.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1686.png,Normal/images/Normal-1686.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1686.png,Normal/masks/Normal-1686.png,True,False,299,299,L,161.58,14fac9b3d76498c3f36b1fd4ba69081a
Normal-1687.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1687.png,Normal/images/Normal-1687.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1687.png,Normal/masks/Normal-1687.png,True,False,299,299,L,108.29,7eaf214315abbbda6a057c0c2b43b5b0
Normal-1688.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1688.png,Normal/images/Normal-1688.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1688.png,Normal/masks/Normal-1688.png,True,False,299,299,L,132.26,df8b36fa39703b40d4a1863af0d3a416
Normal-1689.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1689.png,Normal/images/Normal-1689.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1689.png,Normal/masks/Normal-1689.png,True,False,299,299,L,167.95,4402736cc4e35706e1a4fa11def75bfa
Normal-169.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-169.png,Normal/images/Normal-169.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-169.png,Normal/masks/Normal-169.png,True,False,299,299,L,102.69,86dd22e1256e209d68e18a09644ee492
Normal-1690.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1690.png,Normal/images/Normal-1690.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1690.png,Normal/masks/Normal-1690.png,True,False,299,299,L,145.3,5eb287733f6d00a5e866c0e890291136
Normal-1691.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1691.png,Normal/images/Normal-1691.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1691.png,Normal/masks/Normal-1691.png,True,False,299,299,L,142.6,00ddab9d23a09d3f3978785034138c83
Normal-1692.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1692.png,Normal/images/Normal-1692.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1692.png,Normal/masks/Normal-1692.png,True,False,299,299,L,111.97,f24c5b916264e472ff0270ccb59a83b7
Normal-1693.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1693.png,Normal/images/Normal-1693.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1693.png,Normal/masks/Normal-1693.png,True,False,299,299,L,172.26,02c390de0a1878748eb90decd2f34af0
Normal-1694.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1694.png,Normal/images/Normal-1694.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1694.png,Normal/masks/Normal-1694.png,True,False,299,299,L,161.02,ce02c42c1acff9959330b348daf7dea6
Normal-1695.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1695.png,Normal/images/Normal-1695.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1695.png,Normal/masks/Normal-1695.png,True,False,299,299,L,99.98,c16e6e60107d7b39424242dde7f34d86
Normal-1696.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1696.png,Normal/images/Normal-1696.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1696.png,Normal/masks/Normal-1696.png,True,False,299,299,L,157.14,fd86e02b9518e6ef4f668dc3f696934c
Normal-1697.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1697.png,Normal/images/Normal-1697.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1697.png,Normal/masks/Normal-1697.png,True,False,299,299,L,116.33,4c309d0a90e95aba0afa1c884d2a96f2
Normal-1698.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1698.png,Normal/images/Normal-1698.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1698.png,Normal/masks/Normal-1698.png,True,False,299,299,L,135.17,34b510e0d5706e73379b0a437c742313
Normal-1699.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1699.png,Normal/images/Normal-1699.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1699.png,Normal/masks/Normal-1699.png,True,False,299,299,L,152.51,164fb91f695840915fc503c861d75087
Normal-17.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-17.png,Normal/images/Normal-17.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-17.png,Normal/masks/Normal-17.png,True,False,299,299,L,104.6,89264e6c59134ccf009fe354f388f1ae
Normal-170.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-170.png,Normal/images/Normal-170.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-170.png,Normal/masks/Normal-170.png,True,False,299,299,L,115.73,e2f9fe6492d1e36b0898c3bac1d794db
Normal-1700.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1700.png,Normal/images/Normal-1700.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1700.png,Normal/masks/Normal-1700.png,True,False,299,299,L,80.95,009702ce5ed03e5d35f4ea4b28a45943
Normal-1701.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1701.png,Normal/images/Normal-1701.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1701.png,Normal/masks/Normal-1701.png,True,False,299,299,L,149.82,ac033628bc4b048b42aecfcb1a8d918d
Normal-1702.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1702.png,Normal/images/Normal-1702.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1702.png,Normal/masks/Normal-1702.png,True,False,299,299,L,132.43,a2927ff0212fe99fc115f5671f51a743
Normal-1703.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1703.png,Normal/images/Normal-1703.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1703.png,Normal/masks/Normal-1703.png,True,False,299,299,L,105.13,0cb90224ae0f358992e018f9f079face
Normal-1704.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1704.png,Normal/images/Normal-1704.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1704.png,Normal/masks/Normal-1704.png,True,False,299,299,L,116.76,56679313bb88a81ea13862aab50d83b9
Normal-1705.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1705.png,Normal/images/Normal-1705.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1705.png,Normal/masks/Normal-1705.png,True,False,299,299,L,96.0,de9925ce17c6b5a3d7ac70b72fb86a22
Normal-1706.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1706.png,Normal/images/Normal-1706.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1706.png,Normal/masks/Normal-1706.png,True,False,299,299,L,119.65,367e219346ba5201f0672de647e203e1
Normal-1707.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1707.png,Normal/images/Normal-1707.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1707.png,Normal/masks/Normal-1707.png,True,False,299,299,L,127.34,d92b0a5dd73c67ad42a8c05529c4f334
Normal-1708.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1708.png,Normal/images/Normal-1708.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1708.png,Normal/masks/Normal-1708.png,True,False,299,299,L,117.07,2093eea18df33375d56461d9c6ad7d53
Normal-1709.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1709.png,Normal/images/Normal-1709.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1709.png,Normal/masks/Normal-1709.png,True,False,299,299,L,138.93,feabc88f633b33e3bcd7441f346c3cda
Normal-171.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-171.png,Normal/images/Normal-171.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-171.png,Normal/masks/Normal-171.png,True,False,299,299,L,110.13,52509a12ef0fbd07f69fef27d7a6a38e
Normal-1710.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1710.png,Normal/images/Normal-1710.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1710.png,Normal/masks/Normal-1710.png,True,False,299,299,L,111.18,cf428852b03b6e878ef0c48c0cee62a6
Normal-1711.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1711.png,Normal/images/Normal-1711.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1711.png,Normal/masks/Normal-1711.png,True,False,299,299,L,105.81,97b9b2fc5742e0733168e4ae828357fb
Normal-1712.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1712.png,Normal/images/Normal-1712.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1712.png,Normal/masks/Normal-1712.png,True,False,299,299,L,134.47,00ef6baa1b730a8c58dabd26d867f8fa
Normal-1713.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1713.png,Normal/images/Normal-1713.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1713.png,Normal/masks/Normal-1713.png,True,False,299,299,L,145.66,9dc3a5a7c0d2db106031ed34c11c2a6a
Normal-1714.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1714.png,Normal/images/Normal-1714.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1714.png,Normal/masks/Normal-1714.png,True,False,299,299,L,126.47,f974de3eafc29ad1918e2f1ba5a606c6
Normal-1715.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1715.png,Normal/images/Normal-1715.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1715.png,Normal/masks/Normal-1715.png,True,False,299,299,L,156.03,ca801d7ce6dc828d10746034b0249f19
Normal-1716.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1716.png,Normal/images/Normal-1716.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1716.png,Normal/masks/Normal-1716.png,True,False,299,299,L,164.41,a3c0d282b47de5a72f65784faa51410e
Normal-1717.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1717.png,Normal/images/Normal-1717.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1717.png,Normal/masks/Normal-1717.png,True,False,299,299,L,136.42,72420fdaaadb6cd2fa27096a99aaed7d
Normal-1718.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1718.png,Normal/images/Normal-1718.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1718.png,Normal/masks/Normal-1718.png,True,False,299,299,L,163.64,4f42bc39437daed021443d678870a96f
Normal-1719.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1719.png,Normal/images/Normal-1719.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1719.png,Normal/masks/Normal-1719.png,True,False,299,299,L,123.29,86620ffcdfe4c7910a7b6dff07854ccc
Normal-172.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-172.png,Normal/images/Normal-172.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-172.png,Normal/masks/Normal-172.png,True,False,299,299,L,126.8,66f276ac32282cab46fe7b7fa451e5d4
Normal-1720.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1720.png,Normal/images/Normal-1720.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1720.png,Normal/masks/Normal-1720.png,True,False,299,299,L,128.33,28015ccf8b30c6d719b1be183d6d9dfd
Normal-1721.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1721.png,Normal/images/Normal-1721.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1721.png,Normal/masks/Normal-1721.png,True,False,299,299,L,126.88,f00f12cf29dba53a2a3bac25de69a5de
Normal-1722.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1722.png,Normal/images/Normal-1722.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1722.png,Normal/masks/Normal-1722.png,True,False,299,299,L,122.24,955b252953bcc6453104936984320d97
Normal-1723.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1723.png,Normal/images/Normal-1723.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1723.png,Normal/masks/Normal-1723.png,True,False,299,299,L,94.29,391c6e4cb942eb0a420951a4c9cb4cbe
Normal-1724.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1724.png,Normal/images/Normal-1724.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1724.png,Normal/masks/Normal-1724.png,True,False,299,299,L,106.0,b5552cce98ab5bee6118eb37391277c5
Normal-1725.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1725.png,Normal/images/Normal-1725.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1725.png,Normal/masks/Normal-1725.png,True,False,299,299,L,164.73,d3b9dfe1121db15390be1dfa98eba02a
Normal-1726.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1726.png,Normal/images/Normal-1726.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1726.png,Normal/masks/Normal-1726.png,True,False,299,299,L,116.19,ef89076dedd707bd5eab7605445f88e0
Normal-1727.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1727.png,Normal/images/Normal-1727.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1727.png,Normal/masks/Normal-1727.png,True,False,299,299,L,147.32,6295e0a2b9a50df91630547ef7c02841
Normal-1728.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1728.png,Normal/images/Normal-1728.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1728.png,Normal/masks/Normal-1728.png,True,False,299,299,L,113.84,965c6e4c4281baca0bfb903e2529978e
Normal-1729.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1729.png,Normal/images/Normal-1729.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1729.png,Normal/masks/Normal-1729.png,True,False,299,299,L,122.88,5304308f32a5d101b5a10ce7ee4c9872
Normal-173.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-173.png,Normal/images/Normal-173.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-173.png,Normal/masks/Normal-173.png,True,False,299,299,L,126.1,1c56acca5deb70fab932e80529150da6
Normal-1730.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1730.png,Normal/images/Normal-1730.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1730.png,Normal/masks/Normal-1730.png,True,False,299,299,L,154.45,255a6e2950e79a6cd20e48006957f340
Normal-1731.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1731.png,Normal/images/Normal-1731.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1731.png,Normal/masks/Normal-1731.png,True,False,299,299,L,109.51,39350ceca318169feda87e8774d517dc
Normal-1732.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1732.png,Normal/images/Normal-1732.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1732.png,Normal/masks/Normal-1732.png,True,False,299,299,L,103.82,4e4ea2809a54f3868c12013fe37209ec
Normal-1733.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1733.png,Normal/images/Normal-1733.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1733.png,Normal/masks/Normal-1733.png,True,False,299,299,L,114.37,b29f7e427d0b87a74dd87f6b54534596
Normal-1734.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1734.png,Normal/images/Normal-1734.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1734.png,Normal/masks/Normal-1734.png,True,False,299,299,L,92.61,70efa4f5f008b017b58a0ea67b00519d
Normal-1735.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1735.png,Normal/images/Normal-1735.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1735.png,Normal/masks/Normal-1735.png,True,False,299,299,L,167.28,78906da7bc4fc0563a7aaddd430d7aa2
Normal-1736.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1736.png,Normal/images/Normal-1736.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1736.png,Normal/masks/Normal-1736.png,True,False,299,299,L,160.52,a0ef2fa3880567b7336c8dc856ca0de6
Normal-1737.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1737.png,Normal/images/Normal-1737.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1737.png,Normal/masks/Normal-1737.png,True,False,299,299,L,130.85,87fe66d1aa57b5192e593e7b89c4c587
Normal-1738.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1738.png,Normal/images/Normal-1738.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1738.png,Normal/masks/Normal-1738.png,True,False,299,299,L,173.87,dada6b537036cca75fd788d7ff44c60b
Normal-1739.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1739.png,Normal/images/Normal-1739.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1739.png,Normal/masks/Normal-1739.png,True,False,299,299,L,112.38,59390901f24cd6ada07168a6c7b22833
Normal-174.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-174.png,Normal/images/Normal-174.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-174.png,Normal/masks/Normal-174.png,True,False,299,299,L,127.71,6796f190db44c419682c9256f9ef5c64
Normal-1740.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1740.png,Normal/images/Normal-1740.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1740.png,Normal/masks/Normal-1740.png,True,False,299,299,L,115.94,51ac82ca2edf301d017711f4273df41f
Normal-1741.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1741.png,Normal/images/Normal-1741.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1741.png,Normal/masks/Normal-1741.png,True,False,299,299,L,126.35,36efb69646cc124640159ba26238afc8
Normal-1742.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1742.png,Normal/images/Normal-1742.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1742.png,Normal/masks/Normal-1742.png,True,False,299,299,L,116.22,efe58ddc5a38d533fb2157d03fd8ed0e
Normal-1743.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1743.png,Normal/images/Normal-1743.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1743.png,Normal/masks/Normal-1743.png,True,False,299,299,L,106.29,8c5c649782185b325a5c445c0fec0870
Normal-1744.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1744.png,Normal/images/Normal-1744.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1744.png,Normal/masks/Normal-1744.png,True,False,299,299,L,91.11,52352e684b3fa1be30a81dfbd66ae057
Normal-1745.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1745.png,Normal/images/Normal-1745.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1745.png,Normal/masks/Normal-1745.png,True,False,299,299,L,98.78,230deac37e509c90d0696ee6da31278f
Normal-1746.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1746.png,Normal/images/Normal-1746.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1746.png,Normal/masks/Normal-1746.png,True,False,299,299,L,174.6,27b16320413d30216dab71c36321416d
Normal-1747.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1747.png,Normal/images/Normal-1747.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1747.png,Normal/masks/Normal-1747.png,True,False,299,299,L,161.82,920f0887ed96078fa5120d1c311e62b7
Normal-1748.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1748.png,Normal/images/Normal-1748.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1748.png,Normal/masks/Normal-1748.png,True,False,299,299,L,168.35,5a370aecc763a631b7e0651f464cc086
Normal-1749.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1749.png,Normal/images/Normal-1749.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1749.png,Normal/masks/Normal-1749.png,True,False,299,299,L,155.86,cae1af8879435b23cc0cf15331b00523
Normal-175.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-175.png,Normal/images/Normal-175.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-175.png,Normal/masks/Normal-175.png,True,False,299,299,L,115.87,e4b5f878b321f7af5fbc9345b030003e
Normal-1750.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1750.png,Normal/images/Normal-1750.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1750.png,Normal/masks/Normal-1750.png,True,False,299,299,L,168.25,fdd0f4f90b71b0e87c04c9ad5f046731
Normal-1751.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1751.png,Normal/images/Normal-1751.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1751.png,Normal/masks/Normal-1751.png,True,False,299,299,L,134.15,2fa000b7695137a7904e2a5fad5490b8
Normal-1752.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1752.png,Normal/images/Normal-1752.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1752.png,Normal/masks/Normal-1752.png,True,False,299,299,L,138.98,0e79036e2c8717d0fc675c63da58c54c
Normal-1753.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1753.png,Normal/images/Normal-1753.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1753.png,Normal/masks/Normal-1753.png,True,False,299,299,L,115.64,8f1cd444644490707088393c313afb85
Normal-1754.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1754.png,Normal/images/Normal-1754.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1754.png,Normal/masks/Normal-1754.png,True,False,299,299,L,136.11,68000c55029172fa617902c34ef1bab2
Normal-1755.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1755.png,Normal/images/Normal-1755.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1755.png,Normal/masks/Normal-1755.png,True,False,299,299,L,81.16,bfc59127ed9a9b218d1e0fc0e276e007
Normal-1756.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1756.png,Normal/images/Normal-1756.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1756.png,Normal/masks/Normal-1756.png,True,False,299,299,L,107.67,0af075ba58cf569aa035b2c9ca329c08
Normal-1757.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1757.png,Normal/images/Normal-1757.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1757.png,Normal/masks/Normal-1757.png,True,False,299,299,L,130.95,5930e87af1c519111e6a95f6463fccd5
Normal-1758.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1758.png,Normal/images/Normal-1758.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1758.png,Normal/masks/Normal-1758.png,True,False,299,299,L,118.31,c6c1cdefaf9702765eb50d6b0c75493f
Normal-1759.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1759.png,Normal/images/Normal-1759.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1759.png,Normal/masks/Normal-1759.png,True,False,299,299,L,93.72,bf46b1e3390f3e35b2c8a096c0e09084
Normal-176.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-176.png,Normal/images/Normal-176.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-176.png,Normal/masks/Normal-176.png,True,False,299,299,L,91.92,0938f34cc0ec72971ad33ae84f0ada60
Normal-1760.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1760.png,Normal/images/Normal-1760.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1760.png,Normal/masks/Normal-1760.png,True,False,299,299,L,128.55,4dbeec2e14643231924275632f4b73b6
Normal-1761.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1761.png,Normal/images/Normal-1761.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1761.png,Normal/masks/Normal-1761.png,True,False,299,299,L,122.32,74d6f8845845203a210ff535407b8d59
Normal-1762.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1762.png,Normal/images/Normal-1762.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1762.png,Normal/masks/Normal-1762.png,True,False,299,299,L,164.75,bcc99dcd602aff8c35e0647800363da6
Normal-1763.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1763.png,Normal/images/Normal-1763.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1763.png,Normal/masks/Normal-1763.png,True,False,299,299,L,171.98,89b10ed534abe79f4b4703ed8cfa5463
Normal-1764.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1764.png,Normal/images/Normal-1764.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1764.png,Normal/masks/Normal-1764.png,True,False,299,299,L,112.75,f6148eb0324961abd0d38a7aa4e0aa0e
Normal-1765.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1765.png,Normal/images/Normal-1765.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1765.png,Normal/masks/Normal-1765.png,True,False,299,299,L,126.0,da1670d97d61fc3897c0320bda0fedc5
Normal-1766.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1766.png,Normal/images/Normal-1766.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1766.png,Normal/masks/Normal-1766.png,True,False,299,299,L,124.93,e024703cb245f2280fd4ce5cc8964c83
Normal-1767.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1767.png,Normal/images/Normal-1767.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1767.png,Normal/masks/Normal-1767.png,True,False,299,299,L,120.18,aca85edd7b1d93f20404e4ed79e6758a
Normal-1768.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1768.png,Normal/images/Normal-1768.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1768.png,Normal/masks/Normal-1768.png,True,False,299,299,L,106.72,4da395a7786192bd3ba2c8255bbe497c
Normal-1769.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1769.png,Normal/images/Normal-1769.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1769.png,Normal/masks/Normal-1769.png,True,False,299,299,L,128.75,3f85569376ffc3b932b1dbc98de6ce27
Normal-177.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-177.png,Normal/images/Normal-177.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-177.png,Normal/masks/Normal-177.png,True,False,299,299,L,106.8,1ce0b54d8bc8028c83387f1a33b0e1f6
Normal-1770.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1770.png,Normal/images/Normal-1770.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1770.png,Normal/masks/Normal-1770.png,True,False,299,299,L,110.18,caabfe9f7db5186e5c3cef01509d046d
Normal-1771.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1771.png,Normal/images/Normal-1771.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1771.png,Normal/masks/Normal-1771.png,True,False,299,299,L,155.96,930677879648c8bf1cf43f1ff42162f6
Normal-1772.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1772.png,Normal/images/Normal-1772.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1772.png,Normal/masks/Normal-1772.png,True,False,299,299,L,156.55,8cbd99aa52586b3f7616a4bde7f0ba16
Normal-1773.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1773.png,Normal/images/Normal-1773.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1773.png,Normal/masks/Normal-1773.png,True,False,299,299,L,66.79,85a95f71d1f307230b59cae4ffffaaa0
Normal-1774.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1774.png,Normal/images/Normal-1774.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1774.png,Normal/masks/Normal-1774.png,True,False,299,299,L,116.92,9b9beb862c359edc77708bb3d17d14cc
Normal-1775.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1775.png,Normal/images/Normal-1775.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1775.png,Normal/masks/Normal-1775.png,True,False,299,299,L,124.93,7bbbddc8b9071f7128dbbf414fe20eef
Normal-1776.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1776.png,Normal/images/Normal-1776.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1776.png,Normal/masks/Normal-1776.png,True,False,299,299,L,124.89,545a180caf97f1c15e17a229b89046a9
Normal-1777.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1777.png,Normal/images/Normal-1777.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1777.png,Normal/masks/Normal-1777.png,True,False,299,299,L,119.6,9e7ada4d2dc83a754aa876a7a1eef68b
Normal-1778.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1778.png,Normal/images/Normal-1778.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1778.png,Normal/masks/Normal-1778.png,True,False,299,299,L,152.49,f92e1560a84ae53938b62b685d4a9cfc
Normal-1779.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1779.png,Normal/images/Normal-1779.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1779.png,Normal/masks/Normal-1779.png,True,False,299,299,L,148.31,c67a2c854a8ddb3a1e7eac7ad756982c
Normal-178.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-178.png,Normal/images/Normal-178.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-178.png,Normal/masks/Normal-178.png,True,False,299,299,L,120.67,f8586c65a9e6c484d8862839d77c4f5e
Normal-1780.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1780.png,Normal/images/Normal-1780.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1780.png,Normal/masks/Normal-1780.png,True,False,299,299,L,107.78,078326cdde6becfe30240dec504e18d4
Normal-1781.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1781.png,Normal/images/Normal-1781.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1781.png,Normal/masks/Normal-1781.png,True,False,299,299,L,144.21,e8202024d20a64ea9b3b709299572b7a
Normal-1782.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1782.png,Normal/images/Normal-1782.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1782.png,Normal/masks/Normal-1782.png,True,False,299,299,L,120.65,d3fedfcc08338a1611047ce71213f37e
Normal-1783.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1783.png,Normal/images/Normal-1783.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1783.png,Normal/masks/Normal-1783.png,True,False,299,299,L,123.06,81234ef75ac754c432a365d21b29acf1
Normal-1784.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1784.png,Normal/images/Normal-1784.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1784.png,Normal/masks/Normal-1784.png,True,False,299,299,L,167.31,6bf28d680bda53c2b6bba95874441aa9
Normal-1785.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1785.png,Normal/images/Normal-1785.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1785.png,Normal/masks/Normal-1785.png,True,False,299,299,L,126.3,8384531ef422d6401993b6e99cdce402
Normal-1786.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1786.png,Normal/images/Normal-1786.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1786.png,Normal/masks/Normal-1786.png,True,False,299,299,L,179.95,2c592eca1903ac0874f04f942f047651
Normal-1787.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1787.png,Normal/images/Normal-1787.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1787.png,Normal/masks/Normal-1787.png,True,False,299,299,L,120.19,968bf29f218f15b881ddf381211683cf
Normal-1788.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1788.png,Normal/images/Normal-1788.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1788.png,Normal/masks/Normal-1788.png,True,False,299,299,L,119.55,827a8c6da0404c5e6620fe342e84cfee
Normal-1789.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1789.png,Normal/images/Normal-1789.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1789.png,Normal/masks/Normal-1789.png,True,False,299,299,L,105.06,b57bc8f32e90d7a1c0e86575d0c00116
Normal-179.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-179.png,Normal/images/Normal-179.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-179.png,Normal/masks/Normal-179.png,True,False,299,299,L,114.98,d68c6412cf25d9ae91db27e505cb146f
Normal-1790.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1790.png,Normal/images/Normal-1790.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1790.png,Normal/masks/Normal-1790.png,True,False,299,299,L,140.39,f7516a69e63bd873e60103cd43610139
Normal-1791.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1791.png,Normal/images/Normal-1791.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1791.png,Normal/masks/Normal-1791.png,True,False,299,299,L,110.57,73f63ade52b1b2bcdeab68b1ccbb4758
Normal-1792.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1792.png,Normal/images/Normal-1792.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1792.png,Normal/masks/Normal-1792.png,True,False,299,299,L,130.79,06edf4e1de1e88b7c60d96b5c797f13a
Normal-1793.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1793.png,Normal/images/Normal-1793.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1793.png,Normal/masks/Normal-1793.png,True,False,299,299,L,41.68,26571685bf44f47e46177e16c768f076
Normal-1794.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1794.png,Normal/images/Normal-1794.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1794.png,Normal/masks/Normal-1794.png,True,False,299,299,L,130.17,5845308808ffe78400e5e65daa5294d6
Normal-1795.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1795.png,Normal/images/Normal-1795.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1795.png,Normal/masks/Normal-1795.png,True,False,299,299,L,120.71,7e8680050f950576e004ddcbdebecd09
Normal-1796.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1796.png,Normal/images/Normal-1796.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1796.png,Normal/masks/Normal-1796.png,True,False,299,299,L,175.2,602b46759358418dfac3d595d096fbda
Normal-1797.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1797.png,Normal/images/Normal-1797.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1797.png,Normal/masks/Normal-1797.png,True,False,299,299,L,163.27,927dba53d33495ae52cd1bfc27c5d31a
Normal-1798.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1798.png,Normal/images/Normal-1798.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1798.png,Normal/masks/Normal-1798.png,True,False,299,299,L,110.17,2a1ef86e40637ec123070b55e45c7a36
Normal-1799.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1799.png,Normal/images/Normal-1799.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1799.png,Normal/masks/Normal-1799.png,True,False,299,299,L,128.37,b5350fb99855317965b3a7b7963e4c26
Normal-18.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-18.png,Normal/images/Normal-18.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-18.png,Normal/masks/Normal-18.png,True,False,299,299,L,145.26,238fa4eefd6129cb5a76652d81f873da
Normal-180.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-180.png,Normal/images/Normal-180.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-180.png,Normal/masks/Normal-180.png,True,False,299,299,L,134.74,9761fe8d0196317514e01da1ef5483d1
Normal-1800.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1800.png,Normal/images/Normal-1800.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1800.png,Normal/masks/Normal-1800.png,True,False,299,299,L,100.72,8e1f9bb4a0dd2f0d2a6ffe418ec4b4ee
Normal-1801.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1801.png,Normal/images/Normal-1801.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1801.png,Normal/masks/Normal-1801.png,True,False,299,299,L,125.0,d981905bf915ebfc30977f3fdb8cfe2e
Normal-1802.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1802.png,Normal/images/Normal-1802.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1802.png,Normal/masks/Normal-1802.png,True,False,299,299,L,125.37,f6250566ec7242eb886b0af59e72d5b5
Normal-1803.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1803.png,Normal/images/Normal-1803.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1803.png,Normal/masks/Normal-1803.png,True,False,299,299,L,127.64,f02ba28c5e65cd109f134e2ca7df6351
Normal-1804.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1804.png,Normal/images/Normal-1804.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1804.png,Normal/masks/Normal-1804.png,True,False,299,299,L,147.81,fac425cd1f44e7ee5bd8fbebdb38b84e
Normal-1805.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1805.png,Normal/images/Normal-1805.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1805.png,Normal/masks/Normal-1805.png,True,False,299,299,L,152.03,5f995d021bf5271a1bf60f887c3d2e41
Normal-1806.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1806.png,Normal/images/Normal-1806.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1806.png,Normal/masks/Normal-1806.png,True,False,299,299,L,123.33,2150e72903da81cf3cbbc88110402ee2
Normal-1807.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1807.png,Normal/images/Normal-1807.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1807.png,Normal/masks/Normal-1807.png,True,False,299,299,L,120.68,441b9a24b4ddc9015cfe94fedf4b715b
Normal-1808.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1808.png,Normal/images/Normal-1808.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1808.png,Normal/masks/Normal-1808.png,True,False,299,299,L,133.7,5ec9db59fde3a70483bf58a2b2fe8457
Normal-1809.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1809.png,Normal/images/Normal-1809.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1809.png,Normal/masks/Normal-1809.png,True,False,299,299,L,178.39,ce4cd2a5afbf85fae4d1d67ed10beae3
Normal-181.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-181.png,Normal/images/Normal-181.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-181.png,Normal/masks/Normal-181.png,True,False,299,299,L,108.14,080094b15759768711afbc8faf44807d
Normal-1810.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1810.png,Normal/images/Normal-1810.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1810.png,Normal/masks/Normal-1810.png,True,False,299,299,L,111.73,1e26a7eebddda3644ed358dc65325d2d
Normal-1811.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1811.png,Normal/images/Normal-1811.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1811.png,Normal/masks/Normal-1811.png,True,False,299,299,L,114.76,5659822ca856032d9ebe7d167bb578d6
Normal-1812.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1812.png,Normal/images/Normal-1812.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1812.png,Normal/masks/Normal-1812.png,True,False,299,299,L,126.88,b1ee005303a8d9bf590a089536518679
Normal-1813.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1813.png,Normal/images/Normal-1813.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1813.png,Normal/masks/Normal-1813.png,True,False,299,299,L,120.48,83d8c1efb25571bdb5079c0d094af3cb
Normal-1814.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1814.png,Normal/images/Normal-1814.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1814.png,Normal/masks/Normal-1814.png,True,False,299,299,L,137.55,3cbde454fcf85b1a8f446fb0d08b11d5
Normal-1815.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1815.png,Normal/images/Normal-1815.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1815.png,Normal/masks/Normal-1815.png,True,False,299,299,L,146.86,83549fa8962358a1d8a18d021b47bdc7
Normal-1816.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1816.png,Normal/images/Normal-1816.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1816.png,Normal/masks/Normal-1816.png,True,False,299,299,L,118.1,b1a44dc564bde823ab64fe35d953ef8a
Normal-1817.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1817.png,Normal/images/Normal-1817.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1817.png,Normal/masks/Normal-1817.png,True,False,299,299,L,140.29,66da336827a579a12a352eedd383b125
Normal-1818.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1818.png,Normal/images/Normal-1818.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1818.png,Normal/masks/Normal-1818.png,True,False,299,299,L,119.78,ebc6fffd612382672726b7a663163927
Normal-1819.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1819.png,Normal/images/Normal-1819.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1819.png,Normal/masks/Normal-1819.png,True,False,299,299,L,164.19,734c938f83e4de0704b5e8d6e189b448
Normal-182.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-182.png,Normal/images/Normal-182.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-182.png,Normal/masks/Normal-182.png,True,False,299,299,L,143.14,05336d6c432d63467d97f29a62bfea35
Normal-1820.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1820.png,Normal/images/Normal-1820.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1820.png,Normal/masks/Normal-1820.png,True,False,299,299,L,125.04,aca9771b37cf4db1d12b6dc5041ea2a5
Normal-1821.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1821.png,Normal/images/Normal-1821.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1821.png,Normal/masks/Normal-1821.png,True,False,299,299,L,168.17,b2c1204e031fcfec2135b54db23abd89
Normal-1822.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1822.png,Normal/images/Normal-1822.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1822.png,Normal/masks/Normal-1822.png,True,False,299,299,L,102.88,a07b34c95f05ec1b335efca7554944f4
Normal-1823.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1823.png,Normal/images/Normal-1823.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1823.png,Normal/masks/Normal-1823.png,True,False,299,299,L,176.28,60a155985e1dd5daa24fb71389e18245
Normal-1824.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1824.png,Normal/images/Normal-1824.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1824.png,Normal/masks/Normal-1824.png,True,False,299,299,L,158.83,66f56ae1edc197daafc8ae0c394b250e
Normal-1825.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1825.png,Normal/images/Normal-1825.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1825.png,Normal/masks/Normal-1825.png,True,False,299,299,L,168.96,be7ffe9c4ac12ebd2f6e333824f69c2e
Normal-1826.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1826.png,Normal/images/Normal-1826.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1826.png,Normal/masks/Normal-1826.png,True,False,299,299,L,108.46,65d5cd7c95c573c7b6fed383e69cbe27
Normal-1827.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1827.png,Normal/images/Normal-1827.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1827.png,Normal/masks/Normal-1827.png,True,False,299,299,L,137.97,03f2f6aa8fb2c9f383a3610c1240cf47
Normal-1828.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1828.png,Normal/images/Normal-1828.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1828.png,Normal/masks/Normal-1828.png,True,False,299,299,L,113.27,fdf5a484c1a5588e24d230933e1c67e9
Normal-1829.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1829.png,Normal/images/Normal-1829.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1829.png,Normal/masks/Normal-1829.png,True,False,299,299,L,95.86,c3d437b9bc653dd8f656eddd53afaeb7
Normal-183.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-183.png,Normal/images/Normal-183.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-183.png,Normal/masks/Normal-183.png,True,False,299,299,L,138.52,695c9459a7357fe2b407775430bfbab3
Normal-1830.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1830.png,Normal/images/Normal-1830.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1830.png,Normal/masks/Normal-1830.png,True,False,299,299,L,156.96,c947564c78983d8633c63bbc1974c74b
Normal-1831.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1831.png,Normal/images/Normal-1831.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1831.png,Normal/masks/Normal-1831.png,True,False,299,299,L,134.13,0223958ff51fcafa90bc3affc60df39c
Normal-1832.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1832.png,Normal/images/Normal-1832.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1832.png,Normal/masks/Normal-1832.png,True,False,299,299,L,135.38,13bd1cec2bb682c87929f7b125ef5305
Normal-1833.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1833.png,Normal/images/Normal-1833.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1833.png,Normal/masks/Normal-1833.png,True,False,299,299,L,120.86,7ea93c8530a8535c9a732b614702b539
Normal-1834.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1834.png,Normal/images/Normal-1834.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1834.png,Normal/masks/Normal-1834.png,True,False,299,299,L,137.93,2178a5dd0fa217d8d02379cc10cd0407
Normal-1835.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1835.png,Normal/images/Normal-1835.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1835.png,Normal/masks/Normal-1835.png,True,False,299,299,L,110.53,0fa3ca598603ff3fb0b2a2462df31078
Normal-1836.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1836.png,Normal/images/Normal-1836.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1836.png,Normal/masks/Normal-1836.png,True,False,299,299,L,136.35,4650c08b3cd3cca5585c0b09afcee5a4
Normal-1837.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1837.png,Normal/images/Normal-1837.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1837.png,Normal/masks/Normal-1837.png,True,False,299,299,L,166.8,7e612796cdace7c394352469d0c62563
Normal-1838.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1838.png,Normal/images/Normal-1838.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1838.png,Normal/masks/Normal-1838.png,True,False,299,299,L,161.23,85916814892efc548fd6f7b89050446b
Normal-1839.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1839.png,Normal/images/Normal-1839.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1839.png,Normal/masks/Normal-1839.png,True,False,299,299,L,126.15,a54e9db1822074c7cb653c7dc2b16828
Normal-184.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-184.png,Normal/images/Normal-184.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-184.png,Normal/masks/Normal-184.png,True,False,299,299,L,132.5,6cc84f5af99a14d2db4c9dd117ba0940
Normal-1840.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1840.png,Normal/images/Normal-1840.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1840.png,Normal/masks/Normal-1840.png,True,False,299,299,L,136.75,db394fd6fb2d6b11f760b045698a98fd
Normal-1841.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1841.png,Normal/images/Normal-1841.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1841.png,Normal/masks/Normal-1841.png,True,False,299,299,L,109.31,3d026c6f3db777c48106cb81b0120c55
Normal-1842.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1842.png,Normal/images/Normal-1842.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1842.png,Normal/masks/Normal-1842.png,True,False,299,299,L,111.47,680d195b2e005936c58e87a7590e18ca
Normal-1843.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1843.png,Normal/images/Normal-1843.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1843.png,Normal/masks/Normal-1843.png,True,False,299,299,L,98.11,2fcafd33d905866fcbb452ddc4f7932a
Normal-1844.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1844.png,Normal/images/Normal-1844.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1844.png,Normal/masks/Normal-1844.png,True,False,299,299,L,114.78,f77d57c6677002837b6fb028a891c70f
Normal-1845.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1845.png,Normal/images/Normal-1845.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1845.png,Normal/masks/Normal-1845.png,True,False,299,299,L,135.62,32d0afc3f8c230a16ce18d5daa21f8dd
Normal-1846.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1846.png,Normal/images/Normal-1846.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1846.png,Normal/masks/Normal-1846.png,True,False,299,299,L,132.83,6a94d9dd9541c0ea4556d88a1658445d
Normal-1847.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1847.png,Normal/images/Normal-1847.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1847.png,Normal/masks/Normal-1847.png,True,False,299,299,L,145.45,552c9b535b44c38cf8d269b755201b9f
Normal-1848.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1848.png,Normal/images/Normal-1848.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1848.png,Normal/masks/Normal-1848.png,True,False,299,299,L,153.66,45882af4710cd0c7dc36b1cade3d5ef8
Normal-1849.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1849.png,Normal/images/Normal-1849.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1849.png,Normal/masks/Normal-1849.png,True,False,299,299,L,134.98,18891299983bd86a7469fed925c69b74
Normal-185.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-185.png,Normal/images/Normal-185.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-185.png,Normal/masks/Normal-185.png,True,False,299,299,L,100.81,1f28d788067e869835e4f14d41d47469
Normal-1850.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1850.png,Normal/images/Normal-1850.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1850.png,Normal/masks/Normal-1850.png,True,False,299,299,L,109.57,c58929ca50765a51249c499273d1c5d6
Normal-1851.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1851.png,Normal/images/Normal-1851.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1851.png,Normal/masks/Normal-1851.png,True,False,299,299,L,85.6,a3e90ab74ec3724f4803c326f8761424
Normal-1852.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1852.png,Normal/images/Normal-1852.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1852.png,Normal/masks/Normal-1852.png,True,False,299,299,L,127.14,a4fbaa706eddacfdb961b1d1a82f961e
Normal-1853.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1853.png,Normal/images/Normal-1853.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1853.png,Normal/masks/Normal-1853.png,True,False,299,299,L,135.39,8917af7e50b30b5d4fc39107d8251c64
Normal-1854.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1854.png,Normal/images/Normal-1854.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1854.png,Normal/masks/Normal-1854.png,True,False,299,299,L,105.87,4ff9554e679246c8c2fb1ad467313293
Normal-1855.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1855.png,Normal/images/Normal-1855.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1855.png,Normal/masks/Normal-1855.png,True,False,299,299,L,108.87,5315c1f745e497e743f68e58a25e2fdb
Normal-1856.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1856.png,Normal/images/Normal-1856.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1856.png,Normal/masks/Normal-1856.png,True,False,299,299,L,138.77,2677d43c86a7fdfb907dacf5d7101f0e
Normal-1857.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1857.png,Normal/images/Normal-1857.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1857.png,Normal/masks/Normal-1857.png,True,False,299,299,L,141.89,f68efda165ed652d7253c11ad3a96cc9
Normal-1858.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1858.png,Normal/images/Normal-1858.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1858.png,Normal/masks/Normal-1858.png,True,False,299,299,L,117.71,ca3308acdc21b7aa8bd6ae925ac4644b
Normal-1859.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1859.png,Normal/images/Normal-1859.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1859.png,Normal/masks/Normal-1859.png,True,False,299,299,L,157.15,29892e2749bf03a4175e8a8612867f12
Normal-186.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-186.png,Normal/images/Normal-186.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-186.png,Normal/masks/Normal-186.png,True,False,299,299,L,102.49,6d183b0454ed92f820ee0bc84e12b9fc
Normal-1860.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1860.png,Normal/images/Normal-1860.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1860.png,Normal/masks/Normal-1860.png,True,False,299,299,L,123.16,f4acc7c4454565b7e97dd71db6fcef39
Normal-1861.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1861.png,Normal/images/Normal-1861.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1861.png,Normal/masks/Normal-1861.png,True,False,299,299,L,161.92,b67efa6f784f3009e6fe2148645e5ae2
Normal-1862.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1862.png,Normal/images/Normal-1862.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1862.png,Normal/masks/Normal-1862.png,True,False,299,299,L,106.47,cfc23c6c0501cf57353aa40f51a96299
Normal-1863.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1863.png,Normal/images/Normal-1863.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1863.png,Normal/masks/Normal-1863.png,True,False,299,299,L,120.89,80c6f64213c97812251aa3ccf389f45a
Normal-1864.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1864.png,Normal/images/Normal-1864.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1864.png,Normal/masks/Normal-1864.png,True,False,299,299,L,129.18,06059392b1b1b6595173bf382f8575f4
Normal-1865.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1865.png,Normal/images/Normal-1865.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1865.png,Normal/masks/Normal-1865.png,True,False,299,299,L,113.15,2e9088ac3d564337798155c7a0b6bc31
Normal-1866.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1866.png,Normal/images/Normal-1866.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1866.png,Normal/masks/Normal-1866.png,True,False,299,299,L,129.99,70be8c9c7fa2257d1e066c9a9faf6896
Normal-1867.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1867.png,Normal/images/Normal-1867.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1867.png,Normal/masks/Normal-1867.png,True,False,299,299,L,148.41,0c81c2d8d41335093fa60b5c1aa4f5ca
Normal-1868.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1868.png,Normal/images/Normal-1868.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1868.png,Normal/masks/Normal-1868.png,True,False,299,299,L,125.51,7cd903af1575f7dc7176ca1e0a329ba3
Normal-1869.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1869.png,Normal/images/Normal-1869.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1869.png,Normal/masks/Normal-1869.png,True,False,299,299,L,115.04,5a816c308ca10aed8ed3cc57a0c9476d
Normal-187.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-187.png,Normal/images/Normal-187.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-187.png,Normal/masks/Normal-187.png,True,False,299,299,L,142.38,68e3033499491183397b49c0c089ea7b
Normal-1870.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1870.png,Normal/images/Normal-1870.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1870.png,Normal/masks/Normal-1870.png,True,False,299,299,L,107.59,f67c0b0994cb280b365b4f2de429cdf0
Normal-1871.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1871.png,Normal/images/Normal-1871.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1871.png,Normal/masks/Normal-1871.png,True,False,299,299,L,98.28,1c78e320ff8dd101621783144544c9e6
Normal-1872.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1872.png,Normal/images/Normal-1872.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1872.png,Normal/masks/Normal-1872.png,True,False,299,299,L,143.29,f95cffeae21a75f664ecf3012cf47e47
Normal-1873.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1873.png,Normal/images/Normal-1873.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1873.png,Normal/masks/Normal-1873.png,True,False,299,299,L,117.62,6996a24f25b187bc857c0f7646a49a4b
Normal-1874.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1874.png,Normal/images/Normal-1874.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1874.png,Normal/masks/Normal-1874.png,True,False,299,299,L,103.47,b0cff79a5daabd08bd16884e661e519c
Normal-1875.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1875.png,Normal/images/Normal-1875.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1875.png,Normal/masks/Normal-1875.png,True,False,299,299,L,100.44,98117a65784cf9e469bf1290e09d7689
Normal-1876.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1876.png,Normal/images/Normal-1876.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1876.png,Normal/masks/Normal-1876.png,True,False,299,299,L,125.58,2c2a5e389755b14271035ff8b66a631e
Normal-1877.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1877.png,Normal/images/Normal-1877.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1877.png,Normal/masks/Normal-1877.png,True,False,299,299,L,130.9,262488c281287f4d28b4d72e716e5f8a
Normal-1878.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1878.png,Normal/images/Normal-1878.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1878.png,Normal/masks/Normal-1878.png,True,False,299,299,L,105.14,1f783e78d3cc57c5a7d9032eae7fbdc0
Normal-1879.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1879.png,Normal/images/Normal-1879.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1879.png,Normal/masks/Normal-1879.png,True,False,299,299,L,106.96,94482496169098ab07c4392df549afda
Normal-188.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-188.png,Normal/images/Normal-188.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-188.png,Normal/masks/Normal-188.png,True,False,299,299,L,99.22,0baf0daed7d1830d86e1369f8e6c39ba
Normal-1880.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1880.png,Normal/images/Normal-1880.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1880.png,Normal/masks/Normal-1880.png,True,False,299,299,L,149.34,58d71ba07f5eb1cc9769f6f473ac3c33
Normal-1881.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1881.png,Normal/images/Normal-1881.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1881.png,Normal/masks/Normal-1881.png,True,False,299,299,L,120.61,62edfbbcc124cabc9671266f73468001
Normal-1882.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1882.png,Normal/images/Normal-1882.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1882.png,Normal/masks/Normal-1882.png,True,False,299,299,L,129.67,9697b4a6376214ecc206ab19f1e7e8b3
Normal-1883.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1883.png,Normal/images/Normal-1883.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1883.png,Normal/masks/Normal-1883.png,True,False,299,299,L,171.12,dc921cd1c3eddd3913b8d0ac75f1cf47
Normal-1884.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1884.png,Normal/images/Normal-1884.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1884.png,Normal/masks/Normal-1884.png,True,False,299,299,L,103.75,ef821981d2be509fd43e1dc6011b81d9
Normal-1885.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1885.png,Normal/images/Normal-1885.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1885.png,Normal/masks/Normal-1885.png,True,False,299,299,L,155.39,a81c97619a7708c0cb8a30e780985600
Normal-1886.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1886.png,Normal/images/Normal-1886.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1886.png,Normal/masks/Normal-1886.png,True,False,299,299,L,142.4,bb03f57d41616d2270993e8c32745fbf
Normal-1887.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1887.png,Normal/images/Normal-1887.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1887.png,Normal/masks/Normal-1887.png,True,False,299,299,L,160.43,12c18ef6fa0c1477164ab5c35205d102
Normal-1888.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1888.png,Normal/images/Normal-1888.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1888.png,Normal/masks/Normal-1888.png,True,False,299,299,L,118.18,23964d1250ea85d07d4f9a4575a9df6a
Normal-1889.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1889.png,Normal/images/Normal-1889.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1889.png,Normal/masks/Normal-1889.png,True,False,299,299,L,162.9,6aceb1c8df1f3cd76328d6f5347480dc
Normal-189.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-189.png,Normal/images/Normal-189.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-189.png,Normal/masks/Normal-189.png,True,False,299,299,L,99.39,4ba4765f8ebb2b63daa8c324b96b07c8
Normal-1890.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1890.png,Normal/images/Normal-1890.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1890.png,Normal/masks/Normal-1890.png,True,False,299,299,L,115.8,706b75fdf20caf6754157ac329ea6ac6
Normal-1891.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1891.png,Normal/images/Normal-1891.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1891.png,Normal/masks/Normal-1891.png,True,False,299,299,L,129.1,508f1c6288a0338a102c0f5f2825476d
Normal-1892.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1892.png,Normal/images/Normal-1892.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1892.png,Normal/masks/Normal-1892.png,True,False,299,299,L,116.38,00f45c42705ea083c58717ac0216c035
Normal-1893.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1893.png,Normal/images/Normal-1893.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1893.png,Normal/masks/Normal-1893.png,True,False,299,299,L,106.91,cfa844826dbdec3b9c111400433955cb
Normal-1894.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1894.png,Normal/images/Normal-1894.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1894.png,Normal/masks/Normal-1894.png,True,False,299,299,L,118.7,8e6f6f61218da073db7ae4699073f810
Normal-1895.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1895.png,Normal/images/Normal-1895.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1895.png,Normal/masks/Normal-1895.png,True,False,299,299,L,110.59,1e893976efcd4bb95f0c1af1597d6f76
Normal-1896.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1896.png,Normal/images/Normal-1896.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1896.png,Normal/masks/Normal-1896.png,True,False,299,299,L,143.58,60c3d3bbaacf5fb5c5f4bbb5338f7587
Normal-1897.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1897.png,Normal/images/Normal-1897.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1897.png,Normal/masks/Normal-1897.png,True,False,299,299,L,153.1,ba10233d3f96fc07eb627140bf1d2e78
Normal-1898.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1898.png,Normal/images/Normal-1898.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1898.png,Normal/masks/Normal-1898.png,True,False,299,299,L,129.41,760ebeb2bd893909db4ecb4ea3b59524
Normal-1899.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1899.png,Normal/images/Normal-1899.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1899.png,Normal/masks/Normal-1899.png,True,False,299,299,L,91.0,f1dab16fd62e785b87f573ebbc5b8003
Normal-19.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-19.png,Normal/images/Normal-19.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-19.png,Normal/masks/Normal-19.png,True,False,299,299,L,112.96,f90c31b1d266fedc41e8fd1a299d3016
Normal-190.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-190.png,Normal/images/Normal-190.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-190.png,Normal/masks/Normal-190.png,True,False,299,299,L,120.83,02a49e34c88a6a6cb0b1ba7e8a495c7d
Normal-1900.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1900.png,Normal/images/Normal-1900.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1900.png,Normal/masks/Normal-1900.png,True,False,299,299,L,115.87,9d5fddacb787a18231e5eb58e7954403
Normal-1901.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1901.png,Normal/images/Normal-1901.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1901.png,Normal/masks/Normal-1901.png,True,False,299,299,L,136.16,ae7b0ab7aa0ae8fd993c7c4f931205ce
Normal-1902.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1902.png,Normal/images/Normal-1902.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1902.png,Normal/masks/Normal-1902.png,True,False,299,299,L,174.14,173008088ed819b6ef8635f13a9b47ca
Normal-1903.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1903.png,Normal/images/Normal-1903.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1903.png,Normal/masks/Normal-1903.png,True,False,299,299,L,158.18,e40193b54972bb777070a533d5773f72
Normal-1904.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1904.png,Normal/images/Normal-1904.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1904.png,Normal/masks/Normal-1904.png,True,False,299,299,L,108.56,cb1b5cf2ad6cdb66e1ef23012014e169
Normal-1905.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1905.png,Normal/images/Normal-1905.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1905.png,Normal/masks/Normal-1905.png,True,False,299,299,L,140.23,9ff309050154aa447bee74397b7a7346
Normal-1906.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1906.png,Normal/images/Normal-1906.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1906.png,Normal/masks/Normal-1906.png,True,False,299,299,L,123.09,7a0468d81ed279f9fe1ae0bfca522560
Normal-1907.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1907.png,Normal/images/Normal-1907.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1907.png,Normal/masks/Normal-1907.png,True,False,299,299,L,118.31,257b797c521c51f758e21f1a93bfce98
Normal-1908.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1908.png,Normal/images/Normal-1908.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1908.png,Normal/masks/Normal-1908.png,True,False,299,299,L,109.21,d258b387788216787f8ff1a66656ba00
Normal-1909.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1909.png,Normal/images/Normal-1909.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1909.png,Normal/masks/Normal-1909.png,True,False,299,299,L,149.84,541279cb857d02c976ab6d9df5e5be63
Normal-191.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-191.png,Normal/images/Normal-191.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-191.png,Normal/masks/Normal-191.png,True,False,299,299,L,129.29,74265e47062c2f58767dc3118c42d059
Normal-1910.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1910.png,Normal/images/Normal-1910.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1910.png,Normal/masks/Normal-1910.png,True,False,299,299,L,148.67,4b02ea7463363d727f0daefe4f2f8bc8
Normal-1911.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1911.png,Normal/images/Normal-1911.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1911.png,Normal/masks/Normal-1911.png,True,False,299,299,L,115.69,75f21d021233d5821817462b13403f89
Normal-1912.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1912.png,Normal/images/Normal-1912.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1912.png,Normal/masks/Normal-1912.png,True,False,299,299,L,159.84,c7ff94b31ae16a3c346a6f1d24989b1b
Normal-1913.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1913.png,Normal/images/Normal-1913.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1913.png,Normal/masks/Normal-1913.png,True,False,299,299,L,98.85,f442d7f0db4829fda919f95058f247c4
Normal-1914.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1914.png,Normal/images/Normal-1914.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1914.png,Normal/masks/Normal-1914.png,True,False,299,299,L,116.32,c9e135db38351d82213fd08c85064376
Normal-1915.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1915.png,Normal/images/Normal-1915.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1915.png,Normal/masks/Normal-1915.png,True,False,299,299,L,174.55,72ab6e84df67837e8991fc148e7deb43
Normal-1916.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1916.png,Normal/images/Normal-1916.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1916.png,Normal/masks/Normal-1916.png,True,False,299,299,L,166.32,b417645835c9478cc785e8dde7b4a25b
Normal-1917.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1917.png,Normal/images/Normal-1917.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1917.png,Normal/masks/Normal-1917.png,True,False,299,299,L,131.94,86de1522a4027318c7a9d68ff7726f7b
Normal-1918.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1918.png,Normal/images/Normal-1918.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1918.png,Normal/masks/Normal-1918.png,True,False,299,299,L,116.78,fd39147f44f0627067848fcf7ea86e9e
Normal-1919.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1919.png,Normal/images/Normal-1919.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1919.png,Normal/masks/Normal-1919.png,True,False,299,299,L,125.83,88d6c70343eb4d03da0c5e42f0ff8a0b
Normal-192.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-192.png,Normal/images/Normal-192.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-192.png,Normal/masks/Normal-192.png,True,False,299,299,L,126.61,3b632637df81c5ba2f8e2949aef20d82
Normal-1920.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1920.png,Normal/images/Normal-1920.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1920.png,Normal/masks/Normal-1920.png,True,False,299,299,L,162.65,5a737d8888f9df868e92b9c2a7018f10
Normal-1921.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1921.png,Normal/images/Normal-1921.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1921.png,Normal/masks/Normal-1921.png,True,False,299,299,L,108.85,ac9bb101bab17162e5daa0164e55868c
Normal-1922.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1922.png,Normal/images/Normal-1922.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1922.png,Normal/masks/Normal-1922.png,True,False,299,299,L,126.85,e470c4c4939d1dcf5503a0753e9c4e1d
Normal-1923.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1923.png,Normal/images/Normal-1923.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1923.png,Normal/masks/Normal-1923.png,True,False,299,299,L,116.83,ff08ea1daeb550eeaef03d936e3d52b9
Normal-1924.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1924.png,Normal/images/Normal-1924.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1924.png,Normal/masks/Normal-1924.png,True,False,299,299,L,118.5,837d2ce3b795adeeece78b42d5c020b8
Normal-1925.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1925.png,Normal/images/Normal-1925.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1925.png,Normal/masks/Normal-1925.png,True,False,299,299,L,124.69,94468ed6b2581c366a5a3e899415b260
Normal-1926.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1926.png,Normal/images/Normal-1926.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1926.png,Normal/masks/Normal-1926.png,True,False,299,299,L,163.32,3acf6b2b890078ccd9d4c4de005fed31
Normal-1927.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1927.png,Normal/images/Normal-1927.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1927.png,Normal/masks/Normal-1927.png,True,False,299,299,L,118.22,f364e56b0fa78108929c5f54b46d3261
Normal-1928.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1928.png,Normal/images/Normal-1928.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1928.png,Normal/masks/Normal-1928.png,True,False,299,299,L,124.55,5533f01dea2c666400a25b50cacb1416
Normal-1929.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1929.png,Normal/images/Normal-1929.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1929.png,Normal/masks/Normal-1929.png,True,False,299,299,L,139.42,cb4f0964d19613b56bdec07e07ec961c
Normal-193.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-193.png,Normal/images/Normal-193.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-193.png,Normal/masks/Normal-193.png,True,False,299,299,L,126.69,8892478a5aa7a5756ba6063123ec3619
Normal-1930.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1930.png,Normal/images/Normal-1930.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1930.png,Normal/masks/Normal-1930.png,True,False,299,299,L,161.05,6bc5627b29244b9fabc2b4f102c8c3a0
Normal-1931.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1931.png,Normal/images/Normal-1931.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1931.png,Normal/masks/Normal-1931.png,True,False,299,299,L,158.22,634d7bf704604fa9481775c7b016cc44
Normal-1932.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1932.png,Normal/images/Normal-1932.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1932.png,Normal/masks/Normal-1932.png,True,False,299,299,L,99.64,4a53a7f7924095dbcfbc98576e3525b1
Normal-1933.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1933.png,Normal/images/Normal-1933.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1933.png,Normal/masks/Normal-1933.png,True,False,299,299,L,142.45,55583d3c61a94509cfda2f55ef855d9a
Normal-1934.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1934.png,Normal/images/Normal-1934.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1934.png,Normal/masks/Normal-1934.png,True,False,299,299,L,135.05,1501f4ea61cc853d32ea7c5d119af107
Normal-1935.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1935.png,Normal/images/Normal-1935.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1935.png,Normal/masks/Normal-1935.png,True,False,299,299,L,137.95,7adf4ba346a7876415401a5c4f8d1039
Normal-1936.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1936.png,Normal/images/Normal-1936.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1936.png,Normal/masks/Normal-1936.png,True,False,299,299,L,122.69,de9b4da5f2cf9a4a8debb91563faae46
Normal-1937.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1937.png,Normal/images/Normal-1937.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1937.png,Normal/masks/Normal-1937.png,True,False,299,299,L,92.67,8d3bc59e72e62715de12b7e3a7228f36
Normal-1938.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1938.png,Normal/images/Normal-1938.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1938.png,Normal/masks/Normal-1938.png,True,False,299,299,L,113.39,08f3444fd99a9734a3e40d68db51c13a
Normal-1939.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1939.png,Normal/images/Normal-1939.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1939.png,Normal/masks/Normal-1939.png,True,False,299,299,L,107.78,5147967da7de4e03f2c1eee9d6fa6c33
Normal-194.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-194.png,Normal/images/Normal-194.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-194.png,Normal/masks/Normal-194.png,True,False,299,299,L,119.79,60dabea97194d35503a8f437090ee3d3
Normal-1940.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1940.png,Normal/images/Normal-1940.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1940.png,Normal/masks/Normal-1940.png,True,False,299,299,L,52.26,06f9e83ec9e048746f2884b29ae2a23d
Normal-1941.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1941.png,Normal/images/Normal-1941.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1941.png,Normal/masks/Normal-1941.png,True,False,299,299,L,172.2,8c8a56b41b306775c907ac535435f71e
Normal-1942.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1942.png,Normal/images/Normal-1942.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1942.png,Normal/masks/Normal-1942.png,True,False,299,299,L,119.87,bfaaaa296d54f37a96916ef22eacfc34
Normal-1943.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1943.png,Normal/images/Normal-1943.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1943.png,Normal/masks/Normal-1943.png,True,False,299,299,L,125.16,2479edacf9fd1944d4648539725a986e
Normal-1944.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1944.png,Normal/images/Normal-1944.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1944.png,Normal/masks/Normal-1944.png,True,False,299,299,L,175.7,9ef9c4fc3f18279cadeb57b909cc3aa4
Normal-1945.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1945.png,Normal/images/Normal-1945.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1945.png,Normal/masks/Normal-1945.png,True,False,299,299,L,79.09,18ce280fc8357fae890dddf040d0f29e
Normal-1946.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1946.png,Normal/images/Normal-1946.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1946.png,Normal/masks/Normal-1946.png,True,False,299,299,L,120.54,2f2408c3bc80159f0b2c4e494f40b7cb
Normal-1947.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1947.png,Normal/images/Normal-1947.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1947.png,Normal/masks/Normal-1947.png,True,False,299,299,L,112.25,5079ab5f7f3795b81ddcb6eae4268561
Normal-1948.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1948.png,Normal/images/Normal-1948.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1948.png,Normal/masks/Normal-1948.png,True,False,299,299,L,92.68,d60a8838f9b5c44bf3714146447d3a7b
Normal-1949.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1949.png,Normal/images/Normal-1949.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1949.png,Normal/masks/Normal-1949.png,True,False,299,299,L,106.46,f1f8ded9e43f7423490e5684671bb65e
Normal-195.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-195.png,Normal/images/Normal-195.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-195.png,Normal/masks/Normal-195.png,True,False,299,299,L,118.46,60374cfc97cb558a5a8fdd323d076056
Normal-1950.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1950.png,Normal/images/Normal-1950.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1950.png,Normal/masks/Normal-1950.png,True,False,299,299,L,136.99,816960e789074be3ab2005dcc239b565
Normal-1951.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1951.png,Normal/images/Normal-1951.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1951.png,Normal/masks/Normal-1951.png,True,False,299,299,L,151.74,5659526c3a4ff704e73e51d54c7ce444
Normal-1952.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1952.png,Normal/images/Normal-1952.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1952.png,Normal/masks/Normal-1952.png,True,False,299,299,L,104.79,fccc6c7c1daacba40197cea4fbaf5b55
Normal-1953.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1953.png,Normal/images/Normal-1953.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1953.png,Normal/masks/Normal-1953.png,True,False,299,299,L,121.36,0adcfbfeccf02509e5e0014da7c9dd11
Normal-1954.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1954.png,Normal/images/Normal-1954.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1954.png,Normal/masks/Normal-1954.png,True,False,299,299,L,123.28,07d316178adfd1a56807fc06fe111741
Normal-1955.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1955.png,Normal/images/Normal-1955.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1955.png,Normal/masks/Normal-1955.png,True,False,299,299,L,162.36,a68bd0b10b4a78644db429f95e019307
Normal-1956.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1956.png,Normal/images/Normal-1956.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1956.png,Normal/masks/Normal-1956.png,True,False,299,299,L,171.92,ce5d1ed214a4f7c02adf110dbc4a7923
Normal-1957.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1957.png,Normal/images/Normal-1957.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1957.png,Normal/masks/Normal-1957.png,True,False,299,299,L,107.95,3557a4c2b5697d0f7097c8f81bd80124
Normal-1958.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1958.png,Normal/images/Normal-1958.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1958.png,Normal/masks/Normal-1958.png,True,False,299,299,L,110.85,bfc0766593dceebc89c8566cae339038
Normal-1959.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1959.png,Normal/images/Normal-1959.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1959.png,Normal/masks/Normal-1959.png,True,False,299,299,L,175.16,db190cb55c34a5defe4e1f61e0ba0c4d
Normal-196.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-196.png,Normal/images/Normal-196.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-196.png,Normal/masks/Normal-196.png,True,False,299,299,L,112.81,4c4b321bbde097865959766e5de9b82f
Normal-1960.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1960.png,Normal/images/Normal-1960.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1960.png,Normal/masks/Normal-1960.png,True,False,299,299,L,133.77,e780ca648c7ae20c07e85c297d80ecb4
Normal-1961.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1961.png,Normal/images/Normal-1961.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1961.png,Normal/masks/Normal-1961.png,True,False,299,299,L,146.64,d4946259c1bc67418ea139efb284910c
Normal-1962.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1962.png,Normal/images/Normal-1962.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1962.png,Normal/masks/Normal-1962.png,True,False,299,299,L,153.08,96e01eb3ebaf7d11367fc139060eb008
Normal-1963.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1963.png,Normal/images/Normal-1963.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1963.png,Normal/masks/Normal-1963.png,True,False,299,299,L,105.88,d64d90e41e34fca2a02ac5383f842c55
Normal-1964.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1964.png,Normal/images/Normal-1964.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1964.png,Normal/masks/Normal-1964.png,True,False,299,299,L,123.18,e8794a4c541bce8fcb58f1520e6c417a
Normal-1965.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1965.png,Normal/images/Normal-1965.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1965.png,Normal/masks/Normal-1965.png,True,False,299,299,L,108.18,e5429d728d9758d2c1589f96b1605aad
Normal-1966.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1966.png,Normal/images/Normal-1966.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1966.png,Normal/masks/Normal-1966.png,True,False,299,299,L,141.92,0f2009ab907d437f1d45fc6080bc4b58
Normal-1967.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1967.png,Normal/images/Normal-1967.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1967.png,Normal/masks/Normal-1967.png,True,False,299,299,L,119.79,a4f0ea5863ce78b5745fbce91c4b5cc7
Normal-1968.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1968.png,Normal/images/Normal-1968.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1968.png,Normal/masks/Normal-1968.png,True,False,299,299,L,101.52,a3c37c80f03a2de792385b867c80fa82
Normal-1969.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1969.png,Normal/images/Normal-1969.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1969.png,Normal/masks/Normal-1969.png,True,False,299,299,L,116.78,54ef672b646fe14db29b6feeebe5548d
Normal-197.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-197.png,Normal/images/Normal-197.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-197.png,Normal/masks/Normal-197.png,True,False,299,299,L,108.09,75dcf0ce30efb18d5aa2d8e1a586a5e1
Normal-1970.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1970.png,Normal/images/Normal-1970.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1970.png,Normal/masks/Normal-1970.png,True,False,299,299,L,135.51,67313a39ca349d571ca082ce569e1f17
Normal-1971.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1971.png,Normal/images/Normal-1971.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1971.png,Normal/masks/Normal-1971.png,True,False,299,299,L,124.69,93d7c97cd88f9919f47d790d73b54671
Normal-1972.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1972.png,Normal/images/Normal-1972.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1972.png,Normal/masks/Normal-1972.png,True,False,299,299,L,119.09,b2c1dcde6cda3854dfbdb4c3b6d968cf
Normal-1973.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1973.png,Normal/images/Normal-1973.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1973.png,Normal/masks/Normal-1973.png,True,False,299,299,L,130.09,b58b6dbeb6a705a9f860590397cea894
Normal-1974.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1974.png,Normal/images/Normal-1974.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1974.png,Normal/masks/Normal-1974.png,True,False,299,299,L,141.9,688b13a9182dc348338d64c7cb8c48b4
Normal-1975.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1975.png,Normal/images/Normal-1975.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1975.png,Normal/masks/Normal-1975.png,True,False,299,299,L,130.66,0059289591a392d325962d8c2e7fb983
Normal-1976.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1976.png,Normal/images/Normal-1976.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1976.png,Normal/masks/Normal-1976.png,True,False,299,299,L,116.56,ef7edb1cf395260729b5017ebafc5ed9
Normal-1977.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1977.png,Normal/images/Normal-1977.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1977.png,Normal/masks/Normal-1977.png,True,False,299,299,L,176.46,b13e7cc09c265f4d68d3fb1bb69adf2d
Normal-1978.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1978.png,Normal/images/Normal-1978.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1978.png,Normal/masks/Normal-1978.png,True,False,299,299,L,135.98,1b6df3a1b3c10b4560c9f7a6d913fc88
Normal-1979.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1979.png,Normal/images/Normal-1979.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1979.png,Normal/masks/Normal-1979.png,True,False,299,299,L,134.26,73dae3b839a7d0434b6c32b6210246cf
Normal-198.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-198.png,Normal/images/Normal-198.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-198.png,Normal/masks/Normal-198.png,True,False,299,299,L,131.67,3b523c224d472364c82887f39f0e3081
Normal-1980.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1980.png,Normal/images/Normal-1980.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1980.png,Normal/masks/Normal-1980.png,True,False,299,299,L,113.41,216de78828aaac665607cffb83677713
Normal-1981.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1981.png,Normal/images/Normal-1981.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1981.png,Normal/masks/Normal-1981.png,True,False,299,299,L,144.13,d0600d464fcd708be9c9c9555c89f78a
Normal-1982.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1982.png,Normal/images/Normal-1982.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1982.png,Normal/masks/Normal-1982.png,True,False,299,299,L,110.47,ab67b2a620ad6b2404af8d5e43987219
Normal-1983.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1983.png,Normal/images/Normal-1983.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1983.png,Normal/masks/Normal-1983.png,True,False,299,299,L,175.08,d4d843a411e301affc133236d47565bc
Normal-1984.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1984.png,Normal/images/Normal-1984.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1984.png,Normal/masks/Normal-1984.png,True,False,299,299,L,162.57,da1065361b1219a8864c3e6abf10cce4
Normal-1985.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1985.png,Normal/images/Normal-1985.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1985.png,Normal/masks/Normal-1985.png,True,False,299,299,L,106.71,3f998adcb483ee955adcbc79048f8796
Normal-1986.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1986.png,Normal/images/Normal-1986.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1986.png,Normal/masks/Normal-1986.png,True,False,299,299,L,131.98,6364a6bc67af5ec679542a01268c45af
Normal-1987.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1987.png,Normal/images/Normal-1987.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1987.png,Normal/masks/Normal-1987.png,True,False,299,299,L,133.08,f30a253942f22f1748ccae506622275c
Normal-1988.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1988.png,Normal/images/Normal-1988.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1988.png,Normal/masks/Normal-1988.png,True,False,299,299,L,117.51,acef147e125e6ad0f1ef6193cd468e65
Normal-1989.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1989.png,Normal/images/Normal-1989.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1989.png,Normal/masks/Normal-1989.png,True,False,299,299,L,120.21,727564e30ad691e2f0c14f800abde6be
Normal-199.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-199.png,Normal/images/Normal-199.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-199.png,Normal/masks/Normal-199.png,True,False,299,299,L,138.07,ae47cc835fa6c470eeb384d289122868
Normal-1990.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1990.png,Normal/images/Normal-1990.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1990.png,Normal/masks/Normal-1990.png,True,False,299,299,L,158.86,709492dd76432b750915626c8b5026ec
Normal-1991.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1991.png,Normal/images/Normal-1991.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1991.png,Normal/masks/Normal-1991.png,True,False,299,299,L,107.96,f33ab7c18d1d4e1952b88fbb1226588c
Normal-1992.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1992.png,Normal/images/Normal-1992.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1992.png,Normal/masks/Normal-1992.png,True,False,299,299,L,127.48,ed71e229a849bcb3c093df374415e23d
Normal-1993.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1993.png,Normal/images/Normal-1993.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1993.png,Normal/masks/Normal-1993.png,True,False,299,299,L,161.31,62c1ca0799b4dabf1be175fe546d3cf1
Normal-1994.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1994.png,Normal/images/Normal-1994.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1994.png,Normal/masks/Normal-1994.png,True,False,299,299,L,108.86,9c4e20bf70af8a1b64efd2acd6086eb1
Normal-1995.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1995.png,Normal/images/Normal-1995.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1995.png,Normal/masks/Normal-1995.png,True,False,299,299,L,118.94,abd0e63fde889ea458b8704a4efd24f9
Normal-1996.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1996.png,Normal/images/Normal-1996.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1996.png,Normal/masks/Normal-1996.png,True,False,299,299,L,139.3,729644ba33de1694080ec2ff392637c9
Normal-1997.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1997.png,Normal/images/Normal-1997.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1997.png,Normal/masks/Normal-1997.png,True,False,299,299,L,167.99,de28c52c7d1af4170ea7eada8d60e5df
Normal-1998.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1998.png,Normal/images/Normal-1998.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1998.png,Normal/masks/Normal-1998.png,True,False,299,299,L,107.7,5d3e633d642cb12151dd0bf9057170d9
Normal-1999.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-1999.png,Normal/images/Normal-1999.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-1999.png,Normal/masks/Normal-1999.png,True,False,299,299,L,129.98,a5b9da8eb4c85dc061b2ae8267b24fb9
Normal-2.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2.png,Normal/images/Normal-2.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2.png,Normal/masks/Normal-2.png,True,False,299,299,L,154.69,8235b119f2f546a6621e93b39e889b16
Normal-20.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-20.png,Normal/images/Normal-20.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-20.png,Normal/masks/Normal-20.png,True,False,299,299,L,117.46,40a1dee058c4eddd9d450e342c3e412d
Normal-200.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-200.png,Normal/images/Normal-200.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-200.png,Normal/masks/Normal-200.png,True,False,299,299,L,120.24,f5e2c05f2b1d983deb05eab033e9ed23
Normal-2000.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2000.png,Normal/images/Normal-2000.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2000.png,Normal/masks/Normal-2000.png,True,False,299,299,L,106.88,ab218982ac5c2f6181b13131846c30be
Normal-2001.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2001.png,Normal/images/Normal-2001.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2001.png,Normal/masks/Normal-2001.png,True,False,299,299,L,153.32,0a702eaee3416f4e2c1456501a8580df
Normal-2002.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2002.png,Normal/images/Normal-2002.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2002.png,Normal/masks/Normal-2002.png,True,False,299,299,L,170.98,7ebc9da4be03a18a8307b1eba3bd06f1
Normal-2003.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2003.png,Normal/images/Normal-2003.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2003.png,Normal/masks/Normal-2003.png,True,False,299,299,L,92.32,3514992363f681661c6323e63642c0c6
Normal-2004.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2004.png,Normal/images/Normal-2004.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2004.png,Normal/masks/Normal-2004.png,True,False,299,299,L,139.11,5f673a43e8ff586fb07b729d7607f528
Normal-2005.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2005.png,Normal/images/Normal-2005.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2005.png,Normal/masks/Normal-2005.png,True,False,299,299,L,127.71,faa0f6c9dcf229bff2b5acdd2c96cbbf
Normal-2006.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2006.png,Normal/images/Normal-2006.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2006.png,Normal/masks/Normal-2006.png,True,False,299,299,L,122.08,9c16fdc852f2aa4f7067ff4473de0772
Normal-2007.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2007.png,Normal/images/Normal-2007.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2007.png,Normal/masks/Normal-2007.png,True,False,299,299,L,80.92,f9b3250ae6a394fa4279458dbba110f3
Normal-2008.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2008.png,Normal/images/Normal-2008.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2008.png,Normal/masks/Normal-2008.png,True,False,299,299,L,106.74,bafd119f144fe31d1f09817c67bd89f2
Normal-2009.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2009.png,Normal/images/Normal-2009.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2009.png,Normal/masks/Normal-2009.png,True,False,299,299,L,109.89,a8cf23d58d7ef3635a9d8bea13bb9f0e
Normal-201.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-201.png,Normal/images/Normal-201.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-201.png,Normal/masks/Normal-201.png,True,False,299,299,L,116.18,60cd547ab5b124c3a13f7873498deee3
Normal-2010.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2010.png,Normal/images/Normal-2010.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2010.png,Normal/masks/Normal-2010.png,True,False,299,299,L,134.92,9a086f4a9400863848da16743491b42f
Normal-2011.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2011.png,Normal/images/Normal-2011.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2011.png,Normal/masks/Normal-2011.png,True,False,299,299,L,84.22,95dc6aa7b16a1f84c1330dee8315b9a5
Normal-2012.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2012.png,Normal/images/Normal-2012.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2012.png,Normal/masks/Normal-2012.png,True,False,299,299,L,128.2,66d08eba3d1cd6e43821e0e88f237806
Normal-2013.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2013.png,Normal/images/Normal-2013.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2013.png,Normal/masks/Normal-2013.png,True,False,299,299,L,123.42,ff105328f1bdb4795d1a5c5bfc60c64d
Normal-2014.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2014.png,Normal/images/Normal-2014.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2014.png,Normal/masks/Normal-2014.png,True,False,299,299,L,137.17,68737b2568a0416a4225a704d3132c91
Normal-2015.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2015.png,Normal/images/Normal-2015.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2015.png,Normal/masks/Normal-2015.png,True,False,299,299,L,144.27,47b4dd012418b3f4585b4955d81c5158
Normal-2016.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2016.png,Normal/images/Normal-2016.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2016.png,Normal/masks/Normal-2016.png,True,False,299,299,L,113.84,f1591ad17ab6d0682916252625e85d89
Normal-2017.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2017.png,Normal/images/Normal-2017.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2017.png,Normal/masks/Normal-2017.png,True,False,299,299,L,175.48,c14432f2dcf4f1efcae7fe553b7de9c8
Normal-2018.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2018.png,Normal/images/Normal-2018.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2018.png,Normal/masks/Normal-2018.png,True,False,299,299,L,174.01,e54f75c51ab889e5d5453831d7e4f455
Normal-2019.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2019.png,Normal/images/Normal-2019.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2019.png,Normal/masks/Normal-2019.png,True,False,299,299,L,153.51,cbf78c8e9ed70c4dc08281bd63cfac47
Normal-202.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-202.png,Normal/images/Normal-202.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-202.png,Normal/masks/Normal-202.png,True,False,299,299,L,129.61,2b9952adc686cb240ed515e5738a2242
Normal-2020.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2020.png,Normal/images/Normal-2020.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2020.png,Normal/masks/Normal-2020.png,True,False,299,299,L,134.01,0ea8029175011e1649561228baff939f
Normal-2021.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2021.png,Normal/images/Normal-2021.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2021.png,Normal/masks/Normal-2021.png,True,False,299,299,L,170.78,a83e7cce1705a47d9eaf692570d6b367
Normal-2022.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2022.png,Normal/images/Normal-2022.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2022.png,Normal/masks/Normal-2022.png,True,False,299,299,L,139.82,909cf8b80cc4a574faa0b72a5e37dd02
Normal-2023.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2023.png,Normal/images/Normal-2023.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2023.png,Normal/masks/Normal-2023.png,True,False,299,299,L,171.01,f70378bdfed8eba706746043db2925f2
Normal-2024.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2024.png,Normal/images/Normal-2024.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2024.png,Normal/masks/Normal-2024.png,True,False,299,299,L,176.96,a3797d9895b1bbde0bee0ae2d63ab729
Normal-2025.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2025.png,Normal/images/Normal-2025.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2025.png,Normal/masks/Normal-2025.png,True,False,299,299,L,119.83,8464ec15330fa495dd5a11e5dec7e6f1
Normal-2026.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2026.png,Normal/images/Normal-2026.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2026.png,Normal/masks/Normal-2026.png,True,False,299,299,L,123.98,7fcda4f7ae9f654496323d2aefac56bc
Normal-2027.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2027.png,Normal/images/Normal-2027.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2027.png,Normal/masks/Normal-2027.png,True,False,299,299,L,152.6,c4211a69e9a6ff85f864439e0f27128e
Normal-2028.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2028.png,Normal/images/Normal-2028.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2028.png,Normal/masks/Normal-2028.png,True,False,299,299,L,156.54,a40977f3b43ab4332d5e68bbe213ad83
Normal-2029.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2029.png,Normal/images/Normal-2029.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2029.png,Normal/masks/Normal-2029.png,True,False,299,299,L,120.11,110e01bae505fde3df937aa48e3bfc0c
Normal-203.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-203.png,Normal/images/Normal-203.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-203.png,Normal/masks/Normal-203.png,True,False,299,299,L,124.59,b0cb1f8f9152baaa2f8bb04eb08aa2ad
Normal-2030.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2030.png,Normal/images/Normal-2030.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2030.png,Normal/masks/Normal-2030.png,True,False,299,299,L,160.91,cdcfd302a410d28e811305e702ec5185
Normal-2031.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2031.png,Normal/images/Normal-2031.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2031.png,Normal/masks/Normal-2031.png,True,False,299,299,L,121.1,8962f96a2094b6faa7a32225bf01d11b
Normal-2032.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2032.png,Normal/images/Normal-2032.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2032.png,Normal/masks/Normal-2032.png,True,False,299,299,L,110.68,31fe443292673d20101093af0df30b63
Normal-2033.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2033.png,Normal/images/Normal-2033.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2033.png,Normal/masks/Normal-2033.png,True,False,299,299,L,143.84,c4e24d1152c655d80da5ca7b507e5457
Normal-2034.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2034.png,Normal/images/Normal-2034.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2034.png,Normal/masks/Normal-2034.png,True,False,299,299,L,162.78,e2194e03d73e5ce4eda8a7aad5ff8baa
Normal-2035.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2035.png,Normal/images/Normal-2035.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2035.png,Normal/masks/Normal-2035.png,True,False,299,299,L,131.95,e81be60fc7e753e862c11e4e13f9b317
Normal-2036.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2036.png,Normal/images/Normal-2036.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2036.png,Normal/masks/Normal-2036.png,True,False,299,299,L,103.6,59536aca6004fe841a8a5a095640bb29
Normal-2037.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2037.png,Normal/images/Normal-2037.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2037.png,Normal/masks/Normal-2037.png,True,False,299,299,L,104.72,55fb113f9151dadf11055cd79bcf66d9
Normal-2038.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2038.png,Normal/images/Normal-2038.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2038.png,Normal/masks/Normal-2038.png,True,False,299,299,L,175.73,234c5a53123cc4f4f4a9a803178881ef
Normal-2039.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2039.png,Normal/images/Normal-2039.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2039.png,Normal/masks/Normal-2039.png,True,False,299,299,L,162.16,43f49a35d28aeb50e0743e7b0f501cbc
Normal-204.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-204.png,Normal/images/Normal-204.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-204.png,Normal/masks/Normal-204.png,True,False,299,299,L,112.59,f67cb9c37c58d90ee59b2ce288687b8f
Normal-2040.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2040.png,Normal/images/Normal-2040.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2040.png,Normal/masks/Normal-2040.png,True,False,299,299,L,117.58,2e13b0fc1be47286f4749955bb9d882a
Normal-2041.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2041.png,Normal/images/Normal-2041.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2041.png,Normal/masks/Normal-2041.png,True,False,299,299,L,164.86,53790d5b1d30732fadcce1463d07fadd
Normal-2042.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2042.png,Normal/images/Normal-2042.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2042.png,Normal/masks/Normal-2042.png,True,False,299,299,L,120.84,fcfe44126422e06e96c7617d3b6cf707
Normal-2043.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2043.png,Normal/images/Normal-2043.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2043.png,Normal/masks/Normal-2043.png,True,False,299,299,L,115.14,298da9dbaf1b1178c193a42d2dc7acf2
Normal-2044.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2044.png,Normal/images/Normal-2044.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2044.png,Normal/masks/Normal-2044.png,True,False,299,299,L,112.02,07d8361aa4e993383123f894af5abcd9
Normal-2045.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2045.png,Normal/images/Normal-2045.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2045.png,Normal/masks/Normal-2045.png,True,False,299,299,L,118.24,c8d8344bccff25121d23a70ef26113d8
Normal-2046.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2046.png,Normal/images/Normal-2046.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2046.png,Normal/masks/Normal-2046.png,True,False,299,299,L,126.8,99372006d22984f5883eaa78043accfb
Normal-2047.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2047.png,Normal/images/Normal-2047.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2047.png,Normal/masks/Normal-2047.png,True,False,299,299,L,104.65,0352e4c1bc095fff0be81840797be417
Normal-2048.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2048.png,Normal/images/Normal-2048.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2048.png,Normal/masks/Normal-2048.png,True,False,299,299,L,123.36,b96423888cf0c70d0f41efab0be893dc
Normal-2049.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2049.png,Normal/images/Normal-2049.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2049.png,Normal/masks/Normal-2049.png,True,False,299,299,L,156.12,bf2de8ee002d34fbe49f545dbdcf6c56
Normal-205.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-205.png,Normal/images/Normal-205.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-205.png,Normal/masks/Normal-205.png,True,False,299,299,L,131.81,8a502efa59803017d4c3941b5b7c5aae
Normal-2050.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2050.png,Normal/images/Normal-2050.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2050.png,Normal/masks/Normal-2050.png,True,False,299,299,L,124.76,b97ef60a2545ba5970dbae46cb3f36b8
Normal-2051.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2051.png,Normal/images/Normal-2051.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2051.png,Normal/masks/Normal-2051.png,True,False,299,299,L,152.37,2a69cbdcd1882d98ff9ae810d3d63ca5
Normal-2052.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2052.png,Normal/images/Normal-2052.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2052.png,Normal/masks/Normal-2052.png,True,False,299,299,L,169.19,886e18b9276ae87dd78bbc1867dd5934
Normal-2053.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2053.png,Normal/images/Normal-2053.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2053.png,Normal/masks/Normal-2053.png,True,False,299,299,L,118.91,43408cf4be892211494a2ce01b431b9e
Normal-2054.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2054.png,Normal/images/Normal-2054.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2054.png,Normal/masks/Normal-2054.png,True,False,299,299,L,97.54,296644a9e82b545c615f2fee93413fd6
Normal-2055.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2055.png,Normal/images/Normal-2055.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2055.png,Normal/masks/Normal-2055.png,True,False,299,299,L,138.21,5dc37ce8d61026e295e025b32a414ff0
Normal-2056.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2056.png,Normal/images/Normal-2056.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2056.png,Normal/masks/Normal-2056.png,True,False,299,299,L,105.87,f03944bea85f1c6353c5a2d7cd717230
Normal-2057.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2057.png,Normal/images/Normal-2057.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2057.png,Normal/masks/Normal-2057.png,True,False,299,299,L,138.18,232fde0ea69379a34e3534e88d428e22
Normal-2058.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2058.png,Normal/images/Normal-2058.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2058.png,Normal/masks/Normal-2058.png,True,False,299,299,L,113.59,caf2f10f4e54a212ceb2e271994ddf16
Normal-2059.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2059.png,Normal/images/Normal-2059.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2059.png,Normal/masks/Normal-2059.png,True,False,299,299,L,152.55,7c7172072a5abbbf2161371fcb3e0806
Normal-206.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-206.png,Normal/images/Normal-206.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-206.png,Normal/masks/Normal-206.png,True,False,299,299,L,117.5,a9a5ed93abd16509e4affa0063d23147
Normal-2060.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2060.png,Normal/images/Normal-2060.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2060.png,Normal/masks/Normal-2060.png,True,False,299,299,L,129.12,6edebdca0a864cde4ec99e562ee9c9b5
Normal-2061.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2061.png,Normal/images/Normal-2061.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2061.png,Normal/masks/Normal-2061.png,True,False,299,299,L,92.34,e477e7364709d118a6fb4162080b5399
Normal-2062.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2062.png,Normal/images/Normal-2062.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2062.png,Normal/masks/Normal-2062.png,True,False,299,299,L,100.35,4e310e7bc3db547af673d08602e6fb54
Normal-2063.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2063.png,Normal/images/Normal-2063.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2063.png,Normal/masks/Normal-2063.png,True,False,299,299,L,120.2,1eb2cf1d1bb2d56470f2cd36498b33b4
Normal-2064.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2064.png,Normal/images/Normal-2064.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2064.png,Normal/masks/Normal-2064.png,True,False,299,299,L,128.0,47d3778acafa7193d31bc9b1ed5895c5
Normal-2065.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2065.png,Normal/images/Normal-2065.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2065.png,Normal/masks/Normal-2065.png,True,False,299,299,L,88.32,ff86937bd757b1ba66f395960c23a809
Normal-2066.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2066.png,Normal/images/Normal-2066.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2066.png,Normal/masks/Normal-2066.png,True,False,299,299,L,107.76,01117ae0e7212763e70cc4c3a74531df
Normal-2067.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2067.png,Normal/images/Normal-2067.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2067.png,Normal/masks/Normal-2067.png,True,False,299,299,L,118.93,e7b2549c3fabfed5a76621df6c8ac404
Normal-2068.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2068.png,Normal/images/Normal-2068.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2068.png,Normal/masks/Normal-2068.png,True,False,299,299,L,124.62,d9dec0e1eb6aa662b5d739c17053abc8
Normal-2069.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2069.png,Normal/images/Normal-2069.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2069.png,Normal/masks/Normal-2069.png,True,False,299,299,L,105.37,d66ecf11dca13ac5962a41cd09a37542
Normal-207.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-207.png,Normal/images/Normal-207.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-207.png,Normal/masks/Normal-207.png,True,False,299,299,L,118.9,5ff9ff7ab15b962d3676d42be8a4c45f
Normal-2070.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2070.png,Normal/images/Normal-2070.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2070.png,Normal/masks/Normal-2070.png,True,False,299,299,L,124.14,01e113b29d26126dbb49e677bccf13cf
Normal-2071.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2071.png,Normal/images/Normal-2071.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2071.png,Normal/masks/Normal-2071.png,True,False,299,299,L,157.8,31b885aac5a14babb6377c4428beb9f7
Normal-2072.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2072.png,Normal/images/Normal-2072.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2072.png,Normal/masks/Normal-2072.png,True,False,299,299,L,103.39,0a38e087a2cd6cf0eb4f76bd21c0f727
Normal-2073.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2073.png,Normal/images/Normal-2073.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2073.png,Normal/masks/Normal-2073.png,True,False,299,299,L,123.0,64ebfb84c7e2e5331407b82a328f0f09
Normal-2074.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2074.png,Normal/images/Normal-2074.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2074.png,Normal/masks/Normal-2074.png,True,False,299,299,L,106.43,b0a2a62ebd1382e728bc97616bba7839
Normal-2075.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2075.png,Normal/images/Normal-2075.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2075.png,Normal/masks/Normal-2075.png,True,False,299,299,L,120.27,7164450e0d9534547f4d779597b41ad2
Normal-2076.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2076.png,Normal/images/Normal-2076.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2076.png,Normal/masks/Normal-2076.png,True,False,299,299,L,124.96,2582a1a0b4bf16fb392fcf851f6eed2e
Normal-2077.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2077.png,Normal/images/Normal-2077.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2077.png,Normal/masks/Normal-2077.png,True,False,299,299,L,118.68,bf54a8845f601922a22e15ea85b292d6
Normal-2078.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2078.png,Normal/images/Normal-2078.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2078.png,Normal/masks/Normal-2078.png,True,False,299,299,L,160.55,88745830dc8b9964f3e806e8e435694c
Normal-2079.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2079.png,Normal/images/Normal-2079.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2079.png,Normal/masks/Normal-2079.png,True,False,299,299,L,125.68,a4fad02dfcf66524d2915fa8c554dd9c
Normal-208.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-208.png,Normal/images/Normal-208.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-208.png,Normal/masks/Normal-208.png,True,False,299,299,L,119.53,72dfe21f57da6861344d2ad6f1b9c058
Normal-2080.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2080.png,Normal/images/Normal-2080.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2080.png,Normal/masks/Normal-2080.png,True,False,299,299,L,168.99,7fc20d7a488a0b1a73e5f45494a8ef58
Normal-2081.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2081.png,Normal/images/Normal-2081.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2081.png,Normal/masks/Normal-2081.png,True,False,299,299,L,162.59,fdaf4d140614a6f97eab6bc4e4240423
Normal-2082.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2082.png,Normal/images/Normal-2082.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2082.png,Normal/masks/Normal-2082.png,True,False,299,299,L,119.67,b0808f45028ef1540dabf2aa04997bcd
Normal-2083.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2083.png,Normal/images/Normal-2083.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2083.png,Normal/masks/Normal-2083.png,True,False,299,299,L,88.84,456d81d5dec530ecccd65bfe4ea8982c
Normal-2084.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2084.png,Normal/images/Normal-2084.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2084.png,Normal/masks/Normal-2084.png,True,False,299,299,L,110.74,6e053ad9228444ada7a406c2571eff23
Normal-2085.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2085.png,Normal/images/Normal-2085.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2085.png,Normal/masks/Normal-2085.png,True,False,299,299,L,127.78,8b6467c952336c5e7b71d08ae61dfe67
Normal-2086.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2086.png,Normal/images/Normal-2086.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2086.png,Normal/masks/Normal-2086.png,True,False,299,299,L,140.97,c9aa924460093d77f253cf3b215278e3
Normal-2087.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2087.png,Normal/images/Normal-2087.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2087.png,Normal/masks/Normal-2087.png,True,False,299,299,L,108.65,22421dca9f3826a963c97398651df175
Normal-2088.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2088.png,Normal/images/Normal-2088.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2088.png,Normal/masks/Normal-2088.png,True,False,299,299,L,152.54,eaac6c8a67df43ba5a91ddde232c4f42
Normal-2089.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2089.png,Normal/images/Normal-2089.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2089.png,Normal/masks/Normal-2089.png,True,False,299,299,L,115.3,231d7fd1521c9948891b7da4482e9ba5
Normal-209.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-209.png,Normal/images/Normal-209.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-209.png,Normal/masks/Normal-209.png,True,False,299,299,L,127.43,5a011c91e913629e93bba7f9a79ef631
Normal-2090.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2090.png,Normal/images/Normal-2090.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2090.png,Normal/masks/Normal-2090.png,True,False,299,299,L,164.25,4770866d9b88b5dcbbe2199ee3c5e780
Normal-2091.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2091.png,Normal/images/Normal-2091.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2091.png,Normal/masks/Normal-2091.png,True,False,299,299,L,117.87,82dbc362cedb77cfc2a862eeb507306e
Normal-2092.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2092.png,Normal/images/Normal-2092.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2092.png,Normal/masks/Normal-2092.png,True,False,299,299,L,107.1,39245344be2989164843270313e2f6d1
Normal-2093.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2093.png,Normal/images/Normal-2093.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2093.png,Normal/masks/Normal-2093.png,True,False,299,299,L,115.93,ab3388f0429679230bd60fb65a10e585
Normal-2094.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2094.png,Normal/images/Normal-2094.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2094.png,Normal/masks/Normal-2094.png,True,False,299,299,L,132.08,d019104c359767ff8a4abceef4362108
Normal-2095.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2095.png,Normal/images/Normal-2095.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2095.png,Normal/masks/Normal-2095.png,True,False,299,299,L,71.25,40c501f80ff2a092d3cea124af93f9af
Normal-2096.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2096.png,Normal/images/Normal-2096.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2096.png,Normal/masks/Normal-2096.png,True,False,299,299,L,112.82,3827df8c5b0c3d374dffeba94a0b9e1a
Normal-2097.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2097.png,Normal/images/Normal-2097.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2097.png,Normal/masks/Normal-2097.png,True,False,299,299,L,189.99,c86add939e800bc4866fc11419b21bc9
Normal-2098.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2098.png,Normal/images/Normal-2098.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2098.png,Normal/masks/Normal-2098.png,True,False,299,299,L,127.3,57b702727d3a77ab8df98ec24bc32080
Normal-2099.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2099.png,Normal/images/Normal-2099.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2099.png,Normal/masks/Normal-2099.png,True,False,299,299,L,131.56,6f1c760ba9b17a8d8658f0a614717ced
Normal-21.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-21.png,Normal/images/Normal-21.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-21.png,Normal/masks/Normal-21.png,True,False,299,299,L,143.62,274e19e38e69e0e03737a8339d338eee
Normal-210.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-210.png,Normal/images/Normal-210.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-210.png,Normal/masks/Normal-210.png,True,False,299,299,L,127.16,f8c57a10f314c266d24f8f57de7a4f6d
Normal-2100.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2100.png,Normal/images/Normal-2100.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2100.png,Normal/masks/Normal-2100.png,True,False,299,299,L,98.51,e686078549ac844e0d41b6879a664e26
Normal-2101.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2101.png,Normal/images/Normal-2101.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2101.png,Normal/masks/Normal-2101.png,True,False,299,299,L,126.36,fd4a3507b3a561945a31b24757dfc55a
Normal-2102.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2102.png,Normal/images/Normal-2102.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2102.png,Normal/masks/Normal-2102.png,True,False,299,299,L,113.61,dfa758133eb29ce5eb392292fda24ec8
Normal-2103.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2103.png,Normal/images/Normal-2103.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2103.png,Normal/masks/Normal-2103.png,True,False,299,299,L,156.19,b3795aed2ef1a41e84bcc2370d947904
Normal-2104.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2104.png,Normal/images/Normal-2104.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2104.png,Normal/masks/Normal-2104.png,True,False,299,299,L,142.61,406d11a7388e36a0eda066e333b161de
Normal-2105.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2105.png,Normal/images/Normal-2105.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2105.png,Normal/masks/Normal-2105.png,True,False,299,299,L,126.0,420e747a4e9a38f16fb7e26f0df4733a
Normal-2106.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2106.png,Normal/images/Normal-2106.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2106.png,Normal/masks/Normal-2106.png,True,False,299,299,L,109.73,3120267276a46f4dd90abc3880589c4a
Normal-2107.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2107.png,Normal/images/Normal-2107.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2107.png,Normal/masks/Normal-2107.png,True,False,299,299,L,196.8,e357f2fb182381020af90553d30be84d
Normal-2108.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2108.png,Normal/images/Normal-2108.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2108.png,Normal/masks/Normal-2108.png,True,False,299,299,L,107.61,6092037a37f82062095c7895b404ddfe
Normal-2109.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2109.png,Normal/images/Normal-2109.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2109.png,Normal/masks/Normal-2109.png,True,False,299,299,L,167.36,467bad81a8c45edab588ac7304003279
Normal-211.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-211.png,Normal/images/Normal-211.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-211.png,Normal/masks/Normal-211.png,True,False,299,299,L,129.05,52fb92ef2f152e4b011fbe2fd4dcfac6
Normal-2110.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2110.png,Normal/images/Normal-2110.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2110.png,Normal/masks/Normal-2110.png,True,False,299,299,L,75.49,b379086c005e31da1549fca704801365
Normal-2111.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2111.png,Normal/images/Normal-2111.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2111.png,Normal/masks/Normal-2111.png,True,False,299,299,L,113.86,c03166ef0da9d99d062314a3868e65d4
Normal-2112.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2112.png,Normal/images/Normal-2112.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2112.png,Normal/masks/Normal-2112.png,True,False,299,299,L,112.98,01c2cb42b87ac7f3dfdab22fd897d7cf
Normal-2113.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2113.png,Normal/images/Normal-2113.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2113.png,Normal/masks/Normal-2113.png,True,False,299,299,L,101.56,113d0051e4e15c29416caba6d80414cb
Normal-2114.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2114.png,Normal/images/Normal-2114.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2114.png,Normal/masks/Normal-2114.png,True,False,299,299,L,142.87,7fdaedb0546cfd57d98b7c174d01b60a
Normal-2115.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2115.png,Normal/images/Normal-2115.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2115.png,Normal/masks/Normal-2115.png,True,False,299,299,L,114.4,95167955ae0550aa3d89e4fd4721f35d
Normal-2116.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2116.png,Normal/images/Normal-2116.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2116.png,Normal/masks/Normal-2116.png,True,False,299,299,L,99.63,4327057cd96adaa8dec3018cb9f9817f
Normal-2117.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2117.png,Normal/images/Normal-2117.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2117.png,Normal/masks/Normal-2117.png,True,False,299,299,L,164.82,ecad641ccdfe766ebdd8d4c5b7025e96
Normal-2118.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2118.png,Normal/images/Normal-2118.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2118.png,Normal/masks/Normal-2118.png,True,False,299,299,L,161.71,8621eb7ca00cb9ebb91bc60020ed515c
Normal-2119.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2119.png,Normal/images/Normal-2119.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2119.png,Normal/masks/Normal-2119.png,True,False,299,299,L,142.63,c72a91ae30f8dbef25a05ac812f4a257
Normal-212.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-212.png,Normal/images/Normal-212.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-212.png,Normal/masks/Normal-212.png,True,False,299,299,L,130.8,7983b6819aae89d416f0298a5977024e
Normal-2120.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2120.png,Normal/images/Normal-2120.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2120.png,Normal/masks/Normal-2120.png,True,False,299,299,L,162.35,38ccbbd8b50f81c2bfc826487a076b30
Normal-2121.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2121.png,Normal/images/Normal-2121.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2121.png,Normal/masks/Normal-2121.png,True,False,299,299,L,39.76,994d7fad54a75cd9b4086483de048891
Normal-2122.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2122.png,Normal/images/Normal-2122.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2122.png,Normal/masks/Normal-2122.png,True,False,299,299,L,99.14,ca0ea6f6c1aad56bfff9605ec56e5fa0
Normal-2123.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2123.png,Normal/images/Normal-2123.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2123.png,Normal/masks/Normal-2123.png,True,False,299,299,L,121.83,d2e653b6cab788862b444af1d65bdda9
Normal-2124.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2124.png,Normal/images/Normal-2124.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2124.png,Normal/masks/Normal-2124.png,True,False,299,299,L,141.54,057253e8d199fbf38bf72a81274f67fa
Normal-2125.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2125.png,Normal/images/Normal-2125.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2125.png,Normal/masks/Normal-2125.png,True,False,299,299,L,114.19,9ee94dab7aec4121a8eb5cbdbb54df73
Normal-2126.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2126.png,Normal/images/Normal-2126.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2126.png,Normal/masks/Normal-2126.png,True,False,299,299,L,147.31,d7d04b41a80ec08aa377952cb5f9af78
Normal-2127.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2127.png,Normal/images/Normal-2127.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2127.png,Normal/masks/Normal-2127.png,True,False,299,299,L,119.84,6fa2338cad06b84820d6250b531f8204
Normal-2128.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2128.png,Normal/images/Normal-2128.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2128.png,Normal/masks/Normal-2128.png,True,False,299,299,L,125.55,fc6bee817032abce62f96c8e82b967bd
Normal-2129.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2129.png,Normal/images/Normal-2129.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2129.png,Normal/masks/Normal-2129.png,True,False,299,299,L,140.03,885062a76c7c251637375191588b5850
Normal-213.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-213.png,Normal/images/Normal-213.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-213.png,Normal/masks/Normal-213.png,True,False,299,299,L,128.5,64ea058c9f79033ddd3015ba5040241f
Normal-2130.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2130.png,Normal/images/Normal-2130.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2130.png,Normal/masks/Normal-2130.png,True,False,299,299,L,118.5,c8155e78b6b8a5c92c053e81c5508949
Normal-2131.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2131.png,Normal/images/Normal-2131.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2131.png,Normal/masks/Normal-2131.png,True,False,299,299,L,127.81,0d31236a2570f83354ff8d28a8d13c5e
Normal-2132.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2132.png,Normal/images/Normal-2132.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2132.png,Normal/masks/Normal-2132.png,True,False,299,299,L,129.52,9c399c8845a7af7580a647340f350236
Normal-2133.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2133.png,Normal/images/Normal-2133.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2133.png,Normal/masks/Normal-2133.png,True,False,299,299,L,124.95,a2a3c8415042c787383fd0f589725ff0
Normal-2134.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2134.png,Normal/images/Normal-2134.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2134.png,Normal/masks/Normal-2134.png,True,False,299,299,L,105.92,44b6d2b7c1281d1b6e9ab46e2495b789
Normal-2135.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2135.png,Normal/images/Normal-2135.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2135.png,Normal/masks/Normal-2135.png,True,False,299,299,L,160.37,667c03640de05e8c6066e6ba15abfd0b
Normal-2136.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2136.png,Normal/images/Normal-2136.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2136.png,Normal/masks/Normal-2136.png,True,False,299,299,L,127.13,d2ed9a5ed962a305415f326c2c4cfd4c
Normal-2137.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2137.png,Normal/images/Normal-2137.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2137.png,Normal/masks/Normal-2137.png,True,False,299,299,L,133.9,d7ca4b9790351bec5b4807a034c0b09c
Normal-2138.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2138.png,Normal/images/Normal-2138.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2138.png,Normal/masks/Normal-2138.png,True,False,299,299,L,150.98,dbe716ed737acadb7662b3582c2c2a04
Normal-2139.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2139.png,Normal/images/Normal-2139.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2139.png,Normal/masks/Normal-2139.png,True,False,299,299,L,117.12,6cc582f778c3493174ed42fbae96a7bc
Normal-214.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-214.png,Normal/images/Normal-214.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-214.png,Normal/masks/Normal-214.png,True,False,299,299,L,117.2,d8ba251f54ffccefc882363cb678ae15
Normal-2140.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2140.png,Normal/images/Normal-2140.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2140.png,Normal/masks/Normal-2140.png,True,False,299,299,L,170.89,6f4f6fd384f261081e589a0346d27cc2
Normal-2141.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2141.png,Normal/images/Normal-2141.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2141.png,Normal/masks/Normal-2141.png,True,False,299,299,L,125.8,978b4b569bb1304d23e0ae30214c7157
Normal-2142.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2142.png,Normal/images/Normal-2142.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2142.png,Normal/masks/Normal-2142.png,True,False,299,299,L,129.2,775d50fe2186ed1851cbc8f7cd49cd41
Normal-2143.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2143.png,Normal/images/Normal-2143.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2143.png,Normal/masks/Normal-2143.png,True,False,299,299,L,119.0,2e00cf24931831409a1fe6934fd55bb3
Normal-2144.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2144.png,Normal/images/Normal-2144.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2144.png,Normal/masks/Normal-2144.png,True,False,299,299,L,165.81,e9a14982793cb13645035107f5ac93ad
Normal-2145.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2145.png,Normal/images/Normal-2145.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2145.png,Normal/masks/Normal-2145.png,True,False,299,299,L,99.0,cdf394986bca0cb2c16eaa6dbbabda01
Normal-2146.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2146.png,Normal/images/Normal-2146.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2146.png,Normal/masks/Normal-2146.png,True,False,299,299,L,116.76,a0ee341bfb55ea4266506e17d8688c87
Normal-2147.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2147.png,Normal/images/Normal-2147.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2147.png,Normal/masks/Normal-2147.png,True,False,299,299,L,109.43,5bd2b6bf175a53e3c11e251d774eb464
Normal-2148.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2148.png,Normal/images/Normal-2148.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2148.png,Normal/masks/Normal-2148.png,True,False,299,299,L,87.14,fba4afd877b0b0d293f722288bfb78a4
Normal-2149.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2149.png,Normal/images/Normal-2149.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2149.png,Normal/masks/Normal-2149.png,True,False,299,299,L,123.14,f22cb2b0b10efc64af0b5068d0a7b880
Normal-215.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-215.png,Normal/images/Normal-215.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-215.png,Normal/masks/Normal-215.png,True,False,299,299,L,99.29,79eecea08d10a3d2203be6efb6a53169
Normal-2150.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2150.png,Normal/images/Normal-2150.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2150.png,Normal/masks/Normal-2150.png,True,False,299,299,L,127.05,c3cf0bfbdcc7e811c48d21d2ce73dfc1
Normal-2151.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2151.png,Normal/images/Normal-2151.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2151.png,Normal/masks/Normal-2151.png,True,False,299,299,L,95.11,793936dee80415c2cf76b9425b715da4
Normal-2152.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2152.png,Normal/images/Normal-2152.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2152.png,Normal/masks/Normal-2152.png,True,False,299,299,L,117.37,1fa38342ac76d82097a48790452dc0f4
Normal-2153.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2153.png,Normal/images/Normal-2153.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2153.png,Normal/masks/Normal-2153.png,True,False,299,299,L,99.42,b7cf984b4a25565cb3f328d353b2d467
Normal-2154.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2154.png,Normal/images/Normal-2154.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2154.png,Normal/masks/Normal-2154.png,True,False,299,299,L,120.67,23aa28fd3d37fac5923e7fc8a00d1fc9
Normal-2155.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2155.png,Normal/images/Normal-2155.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2155.png,Normal/masks/Normal-2155.png,True,False,299,299,L,134.04,d4385ab680b77548af0b28b6301b60ae
Normal-2156.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2156.png,Normal/images/Normal-2156.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2156.png,Normal/masks/Normal-2156.png,True,False,299,299,L,181.79,3d3621d2eadbad5e42362a4e92feace6
Normal-2157.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2157.png,Normal/images/Normal-2157.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2157.png,Normal/masks/Normal-2157.png,True,False,299,299,L,114.35,d0e0e95158f92fa6d0456e5e3774a5ba
Normal-2158.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2158.png,Normal/images/Normal-2158.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2158.png,Normal/masks/Normal-2158.png,True,False,299,299,L,135.39,b2756a68039157512d023d95e9781caf
Normal-2159.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2159.png,Normal/images/Normal-2159.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2159.png,Normal/masks/Normal-2159.png,True,False,299,299,L,114.38,2630b1d3d9f1ae902ec407b4af5f4f12
Normal-216.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-216.png,Normal/images/Normal-216.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-216.png,Normal/masks/Normal-216.png,True,False,299,299,L,127.95,bbcd911dac3782402f600a049cafd7ba
Normal-2160.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2160.png,Normal/images/Normal-2160.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2160.png,Normal/masks/Normal-2160.png,True,False,299,299,L,149.18,3d4a1c44f408f1b40634b50b6e49d692
Normal-2161.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2161.png,Normal/images/Normal-2161.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2161.png,Normal/masks/Normal-2161.png,True,False,299,299,L,122.59,420204a07902a8b9fcdb25a79b69372d
Normal-2162.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2162.png,Normal/images/Normal-2162.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2162.png,Normal/masks/Normal-2162.png,True,False,299,299,L,103.79,480ed3d68a400affab23880b9a2581f9
Normal-2163.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2163.png,Normal/images/Normal-2163.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2163.png,Normal/masks/Normal-2163.png,True,False,299,299,L,123.2,b8d0508cea1aba4b7ceb147804e4cbac
Normal-2164.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2164.png,Normal/images/Normal-2164.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2164.png,Normal/masks/Normal-2164.png,True,False,299,299,L,127.89,7410f266a65a35fc8d240392c88781fe
Normal-2165.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2165.png,Normal/images/Normal-2165.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2165.png,Normal/masks/Normal-2165.png,True,False,299,299,L,126.74,5ffde266daa917084af340c89db8837a
Normal-2166.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2166.png,Normal/images/Normal-2166.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2166.png,Normal/masks/Normal-2166.png,True,False,299,299,L,138.19,bd445f7062136753ab9aaf0140840ebe
Normal-2167.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2167.png,Normal/images/Normal-2167.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2167.png,Normal/masks/Normal-2167.png,True,False,299,299,L,118.56,8af10c2aef084ec7751b23e4b94e08d3
Normal-2168.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2168.png,Normal/images/Normal-2168.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2168.png,Normal/masks/Normal-2168.png,True,False,299,299,L,139.39,86e1acba2cbe07a02b821767ce986704
Normal-2169.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2169.png,Normal/images/Normal-2169.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2169.png,Normal/masks/Normal-2169.png,True,False,299,299,L,160.92,6a9c4df2bec9242444d9e0435d9aa3d6
Normal-217.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-217.png,Normal/images/Normal-217.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-217.png,Normal/masks/Normal-217.png,True,False,299,299,L,102.44,9ae2be880df4a2fc403cba18df66027d
Normal-2170.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2170.png,Normal/images/Normal-2170.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2170.png,Normal/masks/Normal-2170.png,True,False,299,299,L,110.67,7a4bda94027fba519a3dd803bfb25f09
Normal-2171.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2171.png,Normal/images/Normal-2171.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2171.png,Normal/masks/Normal-2171.png,True,False,299,299,L,127.12,a6ca9d1a650d81ed47a9c9ac6fa79eb0
Normal-2172.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2172.png,Normal/images/Normal-2172.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2172.png,Normal/masks/Normal-2172.png,True,False,299,299,L,122.21,b1a876a3d5d31dd63b63d1ef645aae50
Normal-2173.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2173.png,Normal/images/Normal-2173.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2173.png,Normal/masks/Normal-2173.png,True,False,299,299,L,98.56,400b6cecf26143c8cb75b55bf8e0fc7a
Normal-2174.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2174.png,Normal/images/Normal-2174.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2174.png,Normal/masks/Normal-2174.png,True,False,299,299,L,107.44,7ea3b0f7efbaa1472e015ff1ea4e9f81
Normal-2175.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2175.png,Normal/images/Normal-2175.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2175.png,Normal/masks/Normal-2175.png,True,False,299,299,L,114.35,6d64df42a4003f9643807030698b0f49
Normal-2176.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2176.png,Normal/images/Normal-2176.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2176.png,Normal/masks/Normal-2176.png,True,False,299,299,L,100.0,e5c795064d2a60206ccdbfe9cdb2d983
Normal-2177.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2177.png,Normal/images/Normal-2177.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2177.png,Normal/masks/Normal-2177.png,True,False,299,299,L,131.03,c237c0afe00db3db9943f17f04f08cd8
Normal-2178.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2178.png,Normal/images/Normal-2178.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2178.png,Normal/masks/Normal-2178.png,True,False,299,299,L,168.89,8c509bc4490609fb8e6c71ad302a46d6
Normal-2179.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2179.png,Normal/images/Normal-2179.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2179.png,Normal/masks/Normal-2179.png,True,False,299,299,L,163.83,b68ef36048aaa56b09a2ddf290aa13a0
Normal-218.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-218.png,Normal/images/Normal-218.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-218.png,Normal/masks/Normal-218.png,True,False,299,299,L,128.18,41c26866f464e99221bc9ec9c1ab281c
Normal-2180.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2180.png,Normal/images/Normal-2180.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2180.png,Normal/masks/Normal-2180.png,True,False,299,299,L,143.84,c91742d252d41f836a05067e08dde236
Normal-2181.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2181.png,Normal/images/Normal-2181.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2181.png,Normal/masks/Normal-2181.png,True,False,299,299,L,173.39,81c440c90b2f61457db73c6383c744d4
Normal-2182.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2182.png,Normal/images/Normal-2182.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2182.png,Normal/masks/Normal-2182.png,True,False,299,299,L,156.39,eb1c50c00719f64fa4f5520ce09b75d3
Normal-2183.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2183.png,Normal/images/Normal-2183.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2183.png,Normal/masks/Normal-2183.png,True,False,299,299,L,112.04,62f79daafc06c1b4346f56271fe154d5
Normal-2184.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2184.png,Normal/images/Normal-2184.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2184.png,Normal/masks/Normal-2184.png,True,False,299,299,L,171.1,e06bf2316292aba5a85ce931e98af075
Normal-2185.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2185.png,Normal/images/Normal-2185.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2185.png,Normal/masks/Normal-2185.png,True,False,299,299,L,156.82,1f813d5f9987683084bdc509214d5f38
Normal-2186.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2186.png,Normal/images/Normal-2186.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2186.png,Normal/masks/Normal-2186.png,True,False,299,299,L,104.97,c5a2e45c9c1599ea08e96313fa95f746
Normal-2187.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2187.png,Normal/images/Normal-2187.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2187.png,Normal/masks/Normal-2187.png,True,False,299,299,L,127.28,f5b9052cfca55f82f718425247a0e009
Normal-2188.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2188.png,Normal/images/Normal-2188.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2188.png,Normal/masks/Normal-2188.png,True,False,299,299,L,118.79,b68c8feede7a5a4a70935bde4c6a4787
Normal-2189.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2189.png,Normal/images/Normal-2189.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2189.png,Normal/masks/Normal-2189.png,True,False,299,299,L,115.01,da559e7af22bd9d9755a09624a447b91
Normal-219.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-219.png,Normal/images/Normal-219.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-219.png,Normal/masks/Normal-219.png,True,False,299,299,L,118.97,92358c55f4733ee9381053a61efe1e45
Normal-2190.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2190.png,Normal/images/Normal-2190.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2190.png,Normal/masks/Normal-2190.png,True,False,299,299,L,107.69,79aabd586c068cbddeb7b5b22bae3159
Normal-2191.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2191.png,Normal/images/Normal-2191.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2191.png,Normal/masks/Normal-2191.png,True,False,299,299,L,123.01,1dff1c6e19d34e2aae18b54d57ca8ea1
Normal-2192.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2192.png,Normal/images/Normal-2192.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2192.png,Normal/masks/Normal-2192.png,True,False,299,299,L,147.5,0b49477dbc9ea7b4144627a981b46753
Normal-2193.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2193.png,Normal/images/Normal-2193.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2193.png,Normal/masks/Normal-2193.png,True,False,299,299,L,138.72,9ab0ccd9a02d1c1909c46b3cb9d38f32
Normal-2194.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2194.png,Normal/images/Normal-2194.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2194.png,Normal/masks/Normal-2194.png,True,False,299,299,L,170.46,e18b33544d57ea61f4efb3e511f4c474
Normal-2195.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2195.png,Normal/images/Normal-2195.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2195.png,Normal/masks/Normal-2195.png,True,False,299,299,L,109.74,b1198959aaa08878a19a01f356d31a12
Normal-2196.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2196.png,Normal/images/Normal-2196.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2196.png,Normal/masks/Normal-2196.png,True,False,299,299,L,97.03,82f415725ab7df31653db16eff776e8d
Normal-2197.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2197.png,Normal/images/Normal-2197.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2197.png,Normal/masks/Normal-2197.png,True,False,299,299,L,146.19,8114950756858f0286d8d303ce2bbcdb
Normal-2198.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2198.png,Normal/images/Normal-2198.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2198.png,Normal/masks/Normal-2198.png,True,False,299,299,L,177.49,f5b3a2d2f5d0c36773671d91480ba024
Normal-2199.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2199.png,Normal/images/Normal-2199.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2199.png,Normal/masks/Normal-2199.png,True,False,299,299,L,113.99,a4600121969ada1d7d17415f6c9dcb9e
Normal-22.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-22.png,Normal/images/Normal-22.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-22.png,Normal/masks/Normal-22.png,True,False,299,299,L,165.89,4b84e9e1bdfcee711c87ab0e5f81743f
Normal-220.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-220.png,Normal/images/Normal-220.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-220.png,Normal/masks/Normal-220.png,True,False,299,299,L,123.1,ff0232761a2bf32251f9fa69dc984a29
Normal-2200.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2200.png,Normal/images/Normal-2200.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2200.png,Normal/masks/Normal-2200.png,True,False,299,299,L,120.69,00e9aada01aabbbbd6577a3e8cba8ffa
Normal-2201.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2201.png,Normal/images/Normal-2201.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2201.png,Normal/masks/Normal-2201.png,True,False,299,299,L,135.32,17d6e071465199652278a68a694ebeda
Normal-2202.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2202.png,Normal/images/Normal-2202.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2202.png,Normal/masks/Normal-2202.png,True,False,299,299,L,100.95,4b576517a36d629b559dd6a8f63a2c64
Normal-2203.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2203.png,Normal/images/Normal-2203.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2203.png,Normal/masks/Normal-2203.png,True,False,299,299,L,92.07,713df57c7aae677e17c376a6470327ec
Normal-2204.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2204.png,Normal/images/Normal-2204.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2204.png,Normal/masks/Normal-2204.png,True,False,299,299,L,124.34,fadaf11d7e41a8ebec08be5ff66eb1c0
Normal-2205.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2205.png,Normal/images/Normal-2205.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2205.png,Normal/masks/Normal-2205.png,True,False,299,299,L,102.94,59e964fd8c0de8cb43ff9f1cf3390eda
Normal-2206.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2206.png,Normal/images/Normal-2206.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2206.png,Normal/masks/Normal-2206.png,True,False,299,299,L,184.42,fc798b2634da9c0ff9f4a28fa4efec7d
Normal-2207.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2207.png,Normal/images/Normal-2207.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2207.png,Normal/masks/Normal-2207.png,True,False,299,299,L,107.62,78e83a0c22df074a8addcbac2f7b7a5c
Normal-2208.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2208.png,Normal/images/Normal-2208.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2208.png,Normal/masks/Normal-2208.png,True,False,299,299,L,131.05,b4aab8a321dafa9499ed02c387384db7
Normal-2209.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2209.png,Normal/images/Normal-2209.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2209.png,Normal/masks/Normal-2209.png,True,False,299,299,L,123.19,16f772d7da0f13dbdf6b3803b1acb50b
Normal-221.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-221.png,Normal/images/Normal-221.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-221.png,Normal/masks/Normal-221.png,True,False,299,299,L,118.18,fe839b214d7a51076decff68259087b6
Normal-2210.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2210.png,Normal/images/Normal-2210.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2210.png,Normal/masks/Normal-2210.png,True,False,299,299,L,112.49,aea522b5a6d3f18d17a95cb742f74145
Normal-2211.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2211.png,Normal/images/Normal-2211.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2211.png,Normal/masks/Normal-2211.png,True,False,299,299,L,128.68,cf7bc078e19ba99428c0b1a481c73e4c
Normal-2212.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2212.png,Normal/images/Normal-2212.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2212.png,Normal/masks/Normal-2212.png,True,False,299,299,L,171.85,9fa16b7f6faba2e458783ecd4183f315
Normal-2213.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2213.png,Normal/images/Normal-2213.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2213.png,Normal/masks/Normal-2213.png,True,False,299,299,L,91.55,bd0db3defea336e15660a18099a629bc
Normal-2214.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2214.png,Normal/images/Normal-2214.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2214.png,Normal/masks/Normal-2214.png,True,False,299,299,L,128.12,c3704afbf84ecf05786baac89c73ad0c
Normal-2215.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2215.png,Normal/images/Normal-2215.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2215.png,Normal/masks/Normal-2215.png,True,False,299,299,L,129.17,b4b81e9e07b78711a0f1be95bd019bda
Normal-2216.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2216.png,Normal/images/Normal-2216.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2216.png,Normal/masks/Normal-2216.png,True,False,299,299,L,116.04,6ede57915e04a897e43e67641e88fa56
Normal-2217.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2217.png,Normal/images/Normal-2217.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2217.png,Normal/masks/Normal-2217.png,True,False,299,299,L,135.31,22d6afb193a712e59fcd1acf5114eee2
Normal-2218.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2218.png,Normal/images/Normal-2218.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2218.png,Normal/masks/Normal-2218.png,True,False,299,299,L,174.46,8d437a801d98927bf23ad0939fd4d234
Normal-2219.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2219.png,Normal/images/Normal-2219.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2219.png,Normal/masks/Normal-2219.png,True,False,299,299,L,135.34,7e88af332ca87432532558b4e935fe42
Normal-222.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-222.png,Normal/images/Normal-222.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-222.png,Normal/masks/Normal-222.png,True,False,299,299,L,134.6,03cd90aa6f606d309ccc894bff8ee7e6
Normal-2220.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2220.png,Normal/images/Normal-2220.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2220.png,Normal/masks/Normal-2220.png,True,False,299,299,L,116.03,36050ff74bf56d6279db68f432722db0
Normal-2221.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2221.png,Normal/images/Normal-2221.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2221.png,Normal/masks/Normal-2221.png,True,False,299,299,L,108.86,1d2f2a497de0b554b916b0b0ae0c9ed4
Normal-2222.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2222.png,Normal/images/Normal-2222.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2222.png,Normal/masks/Normal-2222.png,True,False,299,299,L,107.63,9f3eca4dcbb31c0608ef55809597844b
Normal-2223.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2223.png,Normal/images/Normal-2223.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2223.png,Normal/masks/Normal-2223.png,True,False,299,299,L,126.55,30eccbd25eb66deb9ed273d082998a86
Normal-2224.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2224.png,Normal/images/Normal-2224.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2224.png,Normal/masks/Normal-2224.png,True,False,299,299,L,134.93,49a0803c5d885fa5f1653148467e7700
Normal-2225.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2225.png,Normal/images/Normal-2225.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2225.png,Normal/masks/Normal-2225.png,True,False,299,299,L,177.37,1a930cd6c03199024719494fedc4b77e
Normal-2226.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2226.png,Normal/images/Normal-2226.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2226.png,Normal/masks/Normal-2226.png,True,False,299,299,L,125.62,6a88b4410e34760e98ab2fac8fc9dc45
Normal-2227.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2227.png,Normal/images/Normal-2227.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2227.png,Normal/masks/Normal-2227.png,True,False,299,299,L,101.14,39d13e0c1b9ad35cd298d3f665c99b5e
Normal-2228.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2228.png,Normal/images/Normal-2228.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2228.png,Normal/masks/Normal-2228.png,True,False,299,299,L,100.7,2a6bbabf096997043bb321bbb5012d8a
Normal-2229.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2229.png,Normal/images/Normal-2229.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2229.png,Normal/masks/Normal-2229.png,True,False,299,299,L,116.99,4bf62e398b7c7d4c6adc9ea5466cfa94
Normal-223.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-223.png,Normal/images/Normal-223.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-223.png,Normal/masks/Normal-223.png,True,False,299,299,L,114.94,71e699b347303e7a307f03e6da372c85
Normal-2230.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2230.png,Normal/images/Normal-2230.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2230.png,Normal/masks/Normal-2230.png,True,False,299,299,L,123.29,e4770cf7f923163ff152895afd8ac79f
Normal-2231.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2231.png,Normal/images/Normal-2231.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2231.png,Normal/masks/Normal-2231.png,True,False,299,299,L,132.8,d3874df456d07d3f77564b25cf42bf18
Normal-2232.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2232.png,Normal/images/Normal-2232.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2232.png,Normal/masks/Normal-2232.png,True,False,299,299,L,116.71,a8437a1a627598166b94c0e6e79b0f56
Normal-2233.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2233.png,Normal/images/Normal-2233.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2233.png,Normal/masks/Normal-2233.png,True,False,299,299,L,149.32,434949bc2da176c9a8ebdd919f9a5b5b
Normal-2234.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2234.png,Normal/images/Normal-2234.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2234.png,Normal/masks/Normal-2234.png,True,False,299,299,L,110.27,6b712689bd130588662556ea3e856903
Normal-2235.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2235.png,Normal/images/Normal-2235.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2235.png,Normal/masks/Normal-2235.png,True,False,299,299,L,152.68,497d0148d228f478ebe4b443da2e78da
Normal-2236.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2236.png,Normal/images/Normal-2236.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2236.png,Normal/masks/Normal-2236.png,True,False,299,299,L,98.55,563fa297674ee6f0905a721a7751aa24
Normal-2237.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2237.png,Normal/images/Normal-2237.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2237.png,Normal/masks/Normal-2237.png,True,False,299,299,L,107.24,ce4cd2d46f6e5e32a536703e38d32a04
Normal-2238.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2238.png,Normal/images/Normal-2238.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2238.png,Normal/masks/Normal-2238.png,True,False,299,299,L,162.9,1ca9ac6717efddd3c4ad4567f49a8462
Normal-2239.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2239.png,Normal/images/Normal-2239.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2239.png,Normal/masks/Normal-2239.png,True,False,299,299,L,137.51,8ecd15cf3789828ca27683b3922994b8
Normal-224.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-224.png,Normal/images/Normal-224.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-224.png,Normal/masks/Normal-224.png,True,False,299,299,L,103.37,1cbf8a80b5f3e7f9257059ae0fb8deb1
Normal-2240.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2240.png,Normal/images/Normal-2240.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2240.png,Normal/masks/Normal-2240.png,True,False,299,299,L,124.29,ae1fbb84209b50afceac0a641070df09
Normal-2241.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2241.png,Normal/images/Normal-2241.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2241.png,Normal/masks/Normal-2241.png,True,False,299,299,L,118.73,c2e9878d22b453c66611a18ffa51e4c3
Normal-2242.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2242.png,Normal/images/Normal-2242.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2242.png,Normal/masks/Normal-2242.png,True,False,299,299,L,113.12,006af6c6485aaa96d3e21b7401e940e6
Normal-2243.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2243.png,Normal/images/Normal-2243.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2243.png,Normal/masks/Normal-2243.png,True,False,299,299,L,118.07,b71d271da84905f216bee59933bd1506
Normal-2244.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2244.png,Normal/images/Normal-2244.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2244.png,Normal/masks/Normal-2244.png,True,False,299,299,L,182.22,e62e8179410502a1275537b9877499c3
Normal-2245.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2245.png,Normal/images/Normal-2245.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2245.png,Normal/masks/Normal-2245.png,True,False,299,299,L,116.03,de8b95678cc21710345484aaffbefb43
Normal-2246.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2246.png,Normal/images/Normal-2246.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2246.png,Normal/masks/Normal-2246.png,True,False,299,299,L,125.92,978592126baffaaddd40a23417fc8cd5
Normal-2247.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2247.png,Normal/images/Normal-2247.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2247.png,Normal/masks/Normal-2247.png,True,False,299,299,L,60.9,88294d97529de54b47c0a87ac8fffdf6
Normal-2248.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2248.png,Normal/images/Normal-2248.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2248.png,Normal/masks/Normal-2248.png,True,False,299,299,L,112.28,c75364d61e7e2eefda4beaebca3a5526
Normal-2249.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2249.png,Normal/images/Normal-2249.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2249.png,Normal/masks/Normal-2249.png,True,False,299,299,L,163.81,d60867cd405b0a7e64ccc1edae79f477
Normal-225.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-225.png,Normal/images/Normal-225.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-225.png,Normal/masks/Normal-225.png,True,False,299,299,L,133.22,de4332838a1f6f8241f10eb212cb053a
Normal-2250.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2250.png,Normal/images/Normal-2250.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2250.png,Normal/masks/Normal-2250.png,True,False,299,299,L,109.21,a14206fd944346cc4c8315133a197a4e
Normal-2251.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2251.png,Normal/images/Normal-2251.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2251.png,Normal/masks/Normal-2251.png,True,False,299,299,L,131.54,8240e788ef787d5f0d951446fb4dfa8a
Normal-2252.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2252.png,Normal/images/Normal-2252.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2252.png,Normal/masks/Normal-2252.png,True,False,299,299,L,129.18,30a71a161859d1f4ade6310a987f4b7d
Normal-2253.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2253.png,Normal/images/Normal-2253.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2253.png,Normal/masks/Normal-2253.png,True,False,299,299,L,170.04,423351a8b38b9eeb684a18beeb5102ac
Normal-2254.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2254.png,Normal/images/Normal-2254.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2254.png,Normal/masks/Normal-2254.png,True,False,299,299,L,131.99,ac145a8342220dfc346fe9715ff21e00
Normal-2255.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2255.png,Normal/images/Normal-2255.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2255.png,Normal/masks/Normal-2255.png,True,False,299,299,L,166.01,34b5e2c658744f9ec1d505c5b0f8f5f4
Normal-2256.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2256.png,Normal/images/Normal-2256.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2256.png,Normal/masks/Normal-2256.png,True,False,299,299,L,126.84,d5727b509608f58c019e27d7b003f9cd
Normal-2257.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2257.png,Normal/images/Normal-2257.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2257.png,Normal/masks/Normal-2257.png,True,False,299,299,L,132.4,360909e7de66eb016a45d79147a9500f
Normal-2258.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2258.png,Normal/images/Normal-2258.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2258.png,Normal/masks/Normal-2258.png,True,False,299,299,L,129.48,0ca1e8ac811f642fe97d79b45cb243d0
Normal-2259.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2259.png,Normal/images/Normal-2259.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2259.png,Normal/masks/Normal-2259.png,True,False,299,299,L,115.81,d15391336e58002f1456938ba2df3b3b
Normal-226.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-226.png,Normal/images/Normal-226.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-226.png,Normal/masks/Normal-226.png,True,False,299,299,L,125.11,bac4c497c7466f8c9f7e122b39c3b1fb
Normal-2260.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2260.png,Normal/images/Normal-2260.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2260.png,Normal/masks/Normal-2260.png,True,False,299,299,L,125.17,2165d1783d23b9368607587472a00bff
Normal-2261.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2261.png,Normal/images/Normal-2261.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2261.png,Normal/masks/Normal-2261.png,True,False,299,299,L,116.35,d081abaaa984d067c29232517f797d83
Normal-2262.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2262.png,Normal/images/Normal-2262.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2262.png,Normal/masks/Normal-2262.png,True,False,299,299,L,119.12,e00c41f0938bbb75f5323df84ff241c5
Normal-2263.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2263.png,Normal/images/Normal-2263.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2263.png,Normal/masks/Normal-2263.png,True,False,299,299,L,117.26,21b61fa900c0a635ae22f9eb5643c1d4
Normal-2264.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2264.png,Normal/images/Normal-2264.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2264.png,Normal/masks/Normal-2264.png,True,False,299,299,L,105.27,570cfeede787edf2da5f58f464641557
Normal-2265.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2265.png,Normal/images/Normal-2265.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2265.png,Normal/masks/Normal-2265.png,True,False,299,299,L,106.36,c88a47981e6c4112fda33d692d100dd9
Normal-2266.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2266.png,Normal/images/Normal-2266.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2266.png,Normal/masks/Normal-2266.png,True,False,299,299,L,121.87,44a77b0a0ecd3131a8b7d9256038fcaf
Normal-2267.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2267.png,Normal/images/Normal-2267.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2267.png,Normal/masks/Normal-2267.png,True,False,299,299,L,118.23,368ceec9a86770de82e2fde0f47e9b4c
Normal-2268.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2268.png,Normal/images/Normal-2268.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2268.png,Normal/masks/Normal-2268.png,True,False,299,299,L,141.9,ef33c05f6ee860c9c78e1a260ed974c2
Normal-2269.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2269.png,Normal/images/Normal-2269.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2269.png,Normal/masks/Normal-2269.png,True,False,299,299,L,178.49,89c057f76f23fdcca3c2bdb584379413
Normal-227.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-227.png,Normal/images/Normal-227.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-227.png,Normal/masks/Normal-227.png,True,False,299,299,L,119.99,f88fbe46a2ec1e66238d3c7222bac1a9
Normal-2270.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2270.png,Normal/images/Normal-2270.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2270.png,Normal/masks/Normal-2270.png,True,False,299,299,L,131.12,4b9ab7344aeb094b0a810b7b80997c90
Normal-2271.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2271.png,Normal/images/Normal-2271.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2271.png,Normal/masks/Normal-2271.png,True,False,299,299,L,106.29,5e4b22f54827bb0621a27ff1fc40bc60
Normal-2272.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2272.png,Normal/images/Normal-2272.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2272.png,Normal/masks/Normal-2272.png,True,False,299,299,L,140.51,fae8a0a7dce675454155b27c4c4b5eb6
Normal-2273.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2273.png,Normal/images/Normal-2273.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2273.png,Normal/masks/Normal-2273.png,True,False,299,299,L,98.0,b24d518902d44b4bbcbea05a717b17a7
Normal-2274.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2274.png,Normal/images/Normal-2274.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2274.png,Normal/masks/Normal-2274.png,True,False,299,299,L,131.91,ee405078e052ac2ae98d2cfb138efa4c
Normal-2275.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2275.png,Normal/images/Normal-2275.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2275.png,Normal/masks/Normal-2275.png,True,False,299,299,L,120.98,ba92bbc9ff5d1c8e9b11ba1b51ae858d
Normal-2276.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2276.png,Normal/images/Normal-2276.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2276.png,Normal/masks/Normal-2276.png,True,False,299,299,L,131.72,40b6dd2127c298e2c3a5ec3cb0f49be1
Normal-2277.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2277.png,Normal/images/Normal-2277.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2277.png,Normal/masks/Normal-2277.png,True,False,299,299,L,117.29,2ae36eb606073f2cbcc8a4f69dc7a730
Normal-2278.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2278.png,Normal/images/Normal-2278.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2278.png,Normal/masks/Normal-2278.png,True,False,299,299,L,150.43,deb6325edfc42d7510fe4733a7cf9ac1
Normal-2279.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2279.png,Normal/images/Normal-2279.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2279.png,Normal/masks/Normal-2279.png,True,False,299,299,L,118.78,785907acac996b2a71b85e1695846db4
Normal-228.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-228.png,Normal/images/Normal-228.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-228.png,Normal/masks/Normal-228.png,True,False,299,299,L,121.41,d67dfdc409f65ad82a58d8b22641620a
Normal-2280.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2280.png,Normal/images/Normal-2280.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2280.png,Normal/masks/Normal-2280.png,True,False,299,299,L,100.36,f8e45f4330550e88cc30ae78095e0a12
Normal-2281.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2281.png,Normal/images/Normal-2281.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2281.png,Normal/masks/Normal-2281.png,True,False,299,299,L,111.38,16199def1ebf7a3408457ff5d9530a5b
Normal-2282.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2282.png,Normal/images/Normal-2282.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2282.png,Normal/masks/Normal-2282.png,True,False,299,299,L,116.69,3e44e8d01d24c61b3e940b280faf12ed
Normal-2283.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2283.png,Normal/images/Normal-2283.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2283.png,Normal/masks/Normal-2283.png,True,False,299,299,L,134.13,9c93c1e788f63c0b72f50fb62d3479a9
Normal-2284.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2284.png,Normal/images/Normal-2284.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2284.png,Normal/masks/Normal-2284.png,True,False,299,299,L,158.78,d475ac69bf1eb6c1fa684112c0251e91
Normal-2285.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2285.png,Normal/images/Normal-2285.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2285.png,Normal/masks/Normal-2285.png,True,False,299,299,L,113.22,ebc8592eb79e0bcf39a0cd60e467dbb6
Normal-2286.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2286.png,Normal/images/Normal-2286.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2286.png,Normal/masks/Normal-2286.png,True,False,299,299,L,140.42,1952fcafe1220f27c2238d17ad2e9b50
Normal-2287.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2287.png,Normal/images/Normal-2287.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2287.png,Normal/masks/Normal-2287.png,True,False,299,299,L,146.55,ae8287be749516dfe8d6468a5978ab96
Normal-2288.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2288.png,Normal/images/Normal-2288.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2288.png,Normal/masks/Normal-2288.png,True,False,299,299,L,105.56,817c4300833dc30129193e6b9a711c8f
Normal-2289.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2289.png,Normal/images/Normal-2289.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2289.png,Normal/masks/Normal-2289.png,True,False,299,299,L,107.67,3fe99e852441fe7792968aee6e5934f3
Normal-229.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-229.png,Normal/images/Normal-229.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-229.png,Normal/masks/Normal-229.png,True,False,299,299,L,111.3,94ae5f5202409245204fa02999f30957
Normal-2290.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2290.png,Normal/images/Normal-2290.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2290.png,Normal/masks/Normal-2290.png,True,False,299,299,L,161.02,ec7e9b4c9d34d1612976e54082be9216
Normal-2291.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2291.png,Normal/images/Normal-2291.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2291.png,Normal/masks/Normal-2291.png,True,False,299,299,L,126.32,6d74004cffbcbb225df24ec00bbce3dd
Normal-2292.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2292.png,Normal/images/Normal-2292.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2292.png,Normal/masks/Normal-2292.png,True,False,299,299,L,135.07,abca291367c8fc3691b68be4ac5c0ce2
Normal-2293.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2293.png,Normal/images/Normal-2293.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2293.png,Normal/masks/Normal-2293.png,True,False,299,299,L,102.51,20d44a9de5f17cfcc55fdd993e56980f
Normal-2294.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2294.png,Normal/images/Normal-2294.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2294.png,Normal/masks/Normal-2294.png,True,False,299,299,L,176.44,119b0b52da8fc99dd7dd9a78400cc975
Normal-2295.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2295.png,Normal/images/Normal-2295.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2295.png,Normal/masks/Normal-2295.png,True,False,299,299,L,113.33,29473a3580dd925623b7b7e1dcb6ad83
Normal-2296.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2296.png,Normal/images/Normal-2296.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2296.png,Normal/masks/Normal-2296.png,True,False,299,299,L,104.35,9da9a48b891a8f9d106e370c0d1018d2
Normal-2297.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2297.png,Normal/images/Normal-2297.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2297.png,Normal/masks/Normal-2297.png,True,False,299,299,L,112.4,d4a9f80ad481def3614e9d91c2b1608e
Normal-2298.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2298.png,Normal/images/Normal-2298.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2298.png,Normal/masks/Normal-2298.png,True,False,299,299,L,102.62,43186581a4d0d2565aaebcffbe9bbf93
Normal-2299.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2299.png,Normal/images/Normal-2299.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2299.png,Normal/masks/Normal-2299.png,True,False,299,299,L,122.19,3c7d4716ba886955fd8fd96cbc4fbbec
Normal-23.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-23.png,Normal/images/Normal-23.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-23.png,Normal/masks/Normal-23.png,True,False,299,299,L,134.52,91bf65f8e44ea4d04d0da8e3ca4a142d
Normal-230.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-230.png,Normal/images/Normal-230.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-230.png,Normal/masks/Normal-230.png,True,False,299,299,L,112.96,53087209378c519ab03ff7eb72a2cf5f
Normal-2300.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2300.png,Normal/images/Normal-2300.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2300.png,Normal/masks/Normal-2300.png,True,False,299,299,L,126.14,2100ad2ba362752f2b6d507b06ef6b08
Normal-2301.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2301.png,Normal/images/Normal-2301.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2301.png,Normal/masks/Normal-2301.png,True,False,299,299,L,95.84,e4bccecb4069adc346e4ae8e2582035f
Normal-2302.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2302.png,Normal/images/Normal-2302.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2302.png,Normal/masks/Normal-2302.png,True,False,299,299,L,116.98,b53d44dc8c19214280d3e0012fc03736
Normal-2303.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2303.png,Normal/images/Normal-2303.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2303.png,Normal/masks/Normal-2303.png,True,False,299,299,L,158.42,1e202dee2a24c7d4027fc6729de961d0
Normal-2304.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2304.png,Normal/images/Normal-2304.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2304.png,Normal/masks/Normal-2304.png,True,False,299,299,L,113.34,ef249446bbfb2bc2eed35524e274ef8a
Normal-2305.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2305.png,Normal/images/Normal-2305.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2305.png,Normal/masks/Normal-2305.png,True,False,299,299,L,112.3,118401463660f70ff8853a3cd19f7d6e
Normal-2306.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2306.png,Normal/images/Normal-2306.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2306.png,Normal/masks/Normal-2306.png,True,False,299,299,L,108.01,1cae6f265f131c835707b13d436085e2
Normal-2307.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2307.png,Normal/images/Normal-2307.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2307.png,Normal/masks/Normal-2307.png,True,False,299,299,L,125.7,bf7d1799fb036e6128bef5a8ae84d390
Normal-2308.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2308.png,Normal/images/Normal-2308.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2308.png,Normal/masks/Normal-2308.png,True,False,299,299,L,143.8,9b7a61e400db0b5e014f1f832c0a46fd
Normal-2309.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2309.png,Normal/images/Normal-2309.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2309.png,Normal/masks/Normal-2309.png,True,False,299,299,L,166.77,17dc2719931781be63324008edc94d3f
Normal-231.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-231.png,Normal/images/Normal-231.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-231.png,Normal/masks/Normal-231.png,True,False,299,299,L,122.9,5455313cae150a9f6947d04571e0eb71
Normal-2310.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2310.png,Normal/images/Normal-2310.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2310.png,Normal/masks/Normal-2310.png,True,False,299,299,L,156.79,c2fb818c5327082e498ca2d7fbaa7dad
Normal-2311.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2311.png,Normal/images/Normal-2311.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2311.png,Normal/masks/Normal-2311.png,True,False,299,299,L,166.54,e2b5ae05217dcd98ac576bc2e0a08a79
Normal-2312.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2312.png,Normal/images/Normal-2312.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2312.png,Normal/masks/Normal-2312.png,True,False,299,299,L,109.74,a1bd153a87a633789a610a59e6caf09a
Normal-2313.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2313.png,Normal/images/Normal-2313.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2313.png,Normal/masks/Normal-2313.png,True,False,299,299,L,79.69,a5168761f511cdc843fe79253004da80
Normal-2314.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2314.png,Normal/images/Normal-2314.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2314.png,Normal/masks/Normal-2314.png,True,False,299,299,L,105.13,890dc6bc102b22c49ae4d1cd2175a1d1
Normal-2315.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2315.png,Normal/images/Normal-2315.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2315.png,Normal/masks/Normal-2315.png,True,False,299,299,L,122.47,353e28b061de24ef4e5d02b79eccf0ff
Normal-2316.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2316.png,Normal/images/Normal-2316.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2316.png,Normal/masks/Normal-2316.png,True,False,299,299,L,132.67,41a3cc9997374ea1e6a55f2baabe5687
Normal-2317.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2317.png,Normal/images/Normal-2317.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2317.png,Normal/masks/Normal-2317.png,True,False,299,299,L,124.09,4f428e327a975b9c7913177afd494d2f
Normal-2318.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2318.png,Normal/images/Normal-2318.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2318.png,Normal/masks/Normal-2318.png,True,False,299,299,L,112.85,d9f3d72b7a0f0b0093afd255fa2ca3af
Normal-2319.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2319.png,Normal/images/Normal-2319.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2319.png,Normal/masks/Normal-2319.png,True,False,299,299,L,141.03,dfa262c9253688aec0d2a27f82fa6979
Normal-232.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-232.png,Normal/images/Normal-232.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-232.png,Normal/masks/Normal-232.png,True,False,299,299,L,103.84,19f0685775d59537df192dabedd3edea
Normal-2320.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2320.png,Normal/images/Normal-2320.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2320.png,Normal/masks/Normal-2320.png,True,False,299,299,L,170.56,59d6c205c4cbdb07733464b09039f874
Normal-2321.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2321.png,Normal/images/Normal-2321.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2321.png,Normal/masks/Normal-2321.png,True,False,299,299,L,161.11,94f1570c6f185dcbe04bb40b656632c6
Normal-2322.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2322.png,Normal/images/Normal-2322.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2322.png,Normal/masks/Normal-2322.png,True,False,299,299,L,104.36,a3ca396a56e74c4b3f51404b0080871d
Normal-2323.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2323.png,Normal/images/Normal-2323.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2323.png,Normal/masks/Normal-2323.png,True,False,299,299,L,125.53,a68171b24be0f040a12613fcac1be268
Normal-2324.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2324.png,Normal/images/Normal-2324.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2324.png,Normal/masks/Normal-2324.png,True,False,299,299,L,123.82,2607510e5b5c19b68250293d092fb0af
Normal-2325.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2325.png,Normal/images/Normal-2325.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2325.png,Normal/masks/Normal-2325.png,True,False,299,299,L,132.79,e9cadef163592a67b6e207840bb21568
Normal-2326.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2326.png,Normal/images/Normal-2326.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2326.png,Normal/masks/Normal-2326.png,True,False,299,299,L,156.39,6a53a056bbaa009ba1c8af4ba5313cb1
Normal-2327.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2327.png,Normal/images/Normal-2327.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2327.png,Normal/masks/Normal-2327.png,True,False,299,299,L,146.5,1c6969ebf11a203b1dc7d2e283a201ef
Normal-2328.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2328.png,Normal/images/Normal-2328.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2328.png,Normal/masks/Normal-2328.png,True,False,299,299,L,143.38,65e1816be3b51a956c8c7f95669dbb3b
Normal-2329.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2329.png,Normal/images/Normal-2329.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2329.png,Normal/masks/Normal-2329.png,True,False,299,299,L,97.91,cb1bd444bcba15d000dd9733581047b7
Normal-233.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-233.png,Normal/images/Normal-233.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-233.png,Normal/masks/Normal-233.png,True,False,299,299,L,121.97,ec0777ecd03ceaf8825690a0450957db
Normal-2330.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2330.png,Normal/images/Normal-2330.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2330.png,Normal/masks/Normal-2330.png,True,False,299,299,L,129.97,dc8e39537a86a3c1b795694b598294e6
Normal-2331.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2331.png,Normal/images/Normal-2331.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2331.png,Normal/masks/Normal-2331.png,True,False,299,299,L,101.25,718a84708ab55a730661fce5e6e1ade5
Normal-2332.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2332.png,Normal/images/Normal-2332.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2332.png,Normal/masks/Normal-2332.png,True,False,299,299,L,105.35,6f520744edf8773191889d1762727652
Normal-2333.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2333.png,Normal/images/Normal-2333.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2333.png,Normal/masks/Normal-2333.png,True,False,299,299,L,141.15,68edbf1af399f42bb8b48d768c613898
Normal-2334.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2334.png,Normal/images/Normal-2334.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2334.png,Normal/masks/Normal-2334.png,True,False,299,299,L,166.95,fbb7a5494895e3f27d51644ee735ae50
Normal-2335.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2335.png,Normal/images/Normal-2335.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2335.png,Normal/masks/Normal-2335.png,True,False,299,299,L,119.55,c791e2962bb8847c76fffd8e864f5c37
Normal-2336.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2336.png,Normal/images/Normal-2336.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2336.png,Normal/masks/Normal-2336.png,True,False,299,299,L,110.82,3929084464b17536f07b28e653c1ad66
Normal-2337.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2337.png,Normal/images/Normal-2337.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2337.png,Normal/masks/Normal-2337.png,True,False,299,299,L,156.91,c9ae1cf4b449bd4ac8630c7d89275bd8
Normal-2338.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2338.png,Normal/images/Normal-2338.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2338.png,Normal/masks/Normal-2338.png,True,False,299,299,L,125.18,e9eb2c1944139bba6910b97026119671
Normal-2339.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2339.png,Normal/images/Normal-2339.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2339.png,Normal/masks/Normal-2339.png,True,False,299,299,L,122.73,87abca4a0a6d94e22a07a94fafb9371a
Normal-234.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-234.png,Normal/images/Normal-234.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-234.png,Normal/masks/Normal-234.png,True,False,299,299,L,109.88,522dd59e8529cec32171b30c6c02570f
Normal-2340.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2340.png,Normal/images/Normal-2340.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2340.png,Normal/masks/Normal-2340.png,True,False,299,299,L,165.28,c4e94356fc4f20df4ec279ca765a5213
Normal-2341.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2341.png,Normal/images/Normal-2341.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2341.png,Normal/masks/Normal-2341.png,True,False,299,299,L,106.11,e18bc332af0055642f61f2f9f16c948c
Normal-2342.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2342.png,Normal/images/Normal-2342.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2342.png,Normal/masks/Normal-2342.png,True,False,299,299,L,152.17,b102858705feee1f73b166336a53d958
Normal-2343.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2343.png,Normal/images/Normal-2343.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2343.png,Normal/masks/Normal-2343.png,True,False,299,299,L,104.38,349982a9731ed9ac8ace8ededed7b204
Normal-2344.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2344.png,Normal/images/Normal-2344.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2344.png,Normal/masks/Normal-2344.png,True,False,299,299,L,129.95,91c682bbc7204d518405f79690307468
Normal-2345.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2345.png,Normal/images/Normal-2345.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2345.png,Normal/masks/Normal-2345.png,True,False,299,299,L,110.86,139da33e013a0f73380722d8c443952f
Normal-2346.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2346.png,Normal/images/Normal-2346.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2346.png,Normal/masks/Normal-2346.png,True,False,299,299,L,124.67,fe9531239b9868e0ebf189e8796d2a6e
Normal-2347.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2347.png,Normal/images/Normal-2347.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2347.png,Normal/masks/Normal-2347.png,True,False,299,299,L,171.78,c953cd9ea6e9c2208fed958b2eb5edce
Normal-2348.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2348.png,Normal/images/Normal-2348.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2348.png,Normal/masks/Normal-2348.png,True,False,299,299,L,126.0,04e9f3cb3d9bb0495a5528337c099b26
Normal-2349.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2349.png,Normal/images/Normal-2349.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2349.png,Normal/masks/Normal-2349.png,True,False,299,299,L,133.85,9c558fd2094301c851bad9930bec5022
Normal-235.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-235.png,Normal/images/Normal-235.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-235.png,Normal/masks/Normal-235.png,True,False,299,299,L,118.16,6296ea7c073ce6511e67973c8d416f20
Normal-2350.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2350.png,Normal/images/Normal-2350.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2350.png,Normal/masks/Normal-2350.png,True,False,299,299,L,146.49,12fa5960f0c3da6a3ed04613b99fb1c9
Normal-2351.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2351.png,Normal/images/Normal-2351.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2351.png,Normal/masks/Normal-2351.png,True,False,299,299,L,88.8,ffa1ee747b43c330b9a8c4f0fd0e8ccc
Normal-2352.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2352.png,Normal/images/Normal-2352.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2352.png,Normal/masks/Normal-2352.png,True,False,299,299,L,109.68,4133d701fee50c88ed6f5188ae7d234d
Normal-2353.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2353.png,Normal/images/Normal-2353.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2353.png,Normal/masks/Normal-2353.png,True,False,299,299,L,131.82,89fd8dc09008ca072ec7bad7a9180f61
Normal-2354.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2354.png,Normal/images/Normal-2354.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2354.png,Normal/masks/Normal-2354.png,True,False,299,299,L,157.24,a81aa5df1fbf1615ae653bc0751c6c73
Normal-2355.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2355.png,Normal/images/Normal-2355.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2355.png,Normal/masks/Normal-2355.png,True,False,299,299,L,103.79,002c8bebdec253ef6756aad1fa11c364
Normal-2356.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2356.png,Normal/images/Normal-2356.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2356.png,Normal/masks/Normal-2356.png,True,False,299,299,L,142.26,8223021498f2107a6e3d07dc4ead9c88
Normal-2357.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2357.png,Normal/images/Normal-2357.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2357.png,Normal/masks/Normal-2357.png,True,False,299,299,L,138.37,501f5440d9e6c658e215b542af5057e1
Normal-2358.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2358.png,Normal/images/Normal-2358.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2358.png,Normal/masks/Normal-2358.png,True,False,299,299,L,94.5,8c7a08d8c7820ccc87f76c3add49b8d4
Normal-2359.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2359.png,Normal/images/Normal-2359.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2359.png,Normal/masks/Normal-2359.png,True,False,299,299,L,124.49,d7f649442e78c644916dd3f6144c92d0
Normal-236.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-236.png,Normal/images/Normal-236.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-236.png,Normal/masks/Normal-236.png,True,False,299,299,L,121.36,b862e9b2817deae9b5ba7166fb3a69f5
Normal-2360.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2360.png,Normal/images/Normal-2360.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2360.png,Normal/masks/Normal-2360.png,True,False,299,299,L,106.75,dc82d48e00df11f9b5e0ee7db0d240b3
Normal-2361.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2361.png,Normal/images/Normal-2361.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2361.png,Normal/masks/Normal-2361.png,True,False,299,299,L,117.11,9ea2d3ab52959e5684e1e6e829462c9f
Normal-2362.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2362.png,Normal/images/Normal-2362.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2362.png,Normal/masks/Normal-2362.png,True,False,299,299,L,152.79,e99535e85495257c7713997eb1ada7cc
Normal-2363.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2363.png,Normal/images/Normal-2363.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2363.png,Normal/masks/Normal-2363.png,True,False,299,299,L,110.93,2db53ffe7a23317f3c3e0bf892069f00
Normal-2364.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2364.png,Normal/images/Normal-2364.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2364.png,Normal/masks/Normal-2364.png,True,False,299,299,L,137.83,761db72c400e2bb0caf1707339c807cd
Normal-2365.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2365.png,Normal/images/Normal-2365.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2365.png,Normal/masks/Normal-2365.png,True,False,299,299,L,129.16,1ef7e721dbceff9f9aa1670bf882d1e2
Normal-2366.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2366.png,Normal/images/Normal-2366.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2366.png,Normal/masks/Normal-2366.png,True,False,299,299,L,103.94,3dc0e55996cc3bb4ac9e202fa0570ad1
Normal-2367.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2367.png,Normal/images/Normal-2367.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2367.png,Normal/masks/Normal-2367.png,True,False,299,299,L,128.36,968afd49c25cbcd57dfd8e94759f0b0d
Normal-2368.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2368.png,Normal/images/Normal-2368.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2368.png,Normal/masks/Normal-2368.png,True,False,299,299,L,113.05,c4061e9f96a06e580fe92f92c7fa49e3
Normal-2369.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2369.png,Normal/images/Normal-2369.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2369.png,Normal/masks/Normal-2369.png,True,False,299,299,L,111.83,92dd53fce3da18451e4cfc454a710c4e
Normal-237.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-237.png,Normal/images/Normal-237.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-237.png,Normal/masks/Normal-237.png,True,False,299,299,L,125.47,435c86f6b54a20f12e8e012b359c2fd3
Normal-2370.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2370.png,Normal/images/Normal-2370.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2370.png,Normal/masks/Normal-2370.png,True,False,299,299,L,177.57,ab9632e794a7678fc72d457143e9cd9a
Normal-2371.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2371.png,Normal/images/Normal-2371.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2371.png,Normal/masks/Normal-2371.png,True,False,299,299,L,153.53,933e2be7766f3932241f1f0f1b9ab104
Normal-2372.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2372.png,Normal/images/Normal-2372.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2372.png,Normal/masks/Normal-2372.png,True,False,299,299,L,143.07,8ff6d35b016e0a516d097ede07868bd9
Normal-2373.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2373.png,Normal/images/Normal-2373.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2373.png,Normal/masks/Normal-2373.png,True,False,299,299,L,163.93,80dc47b60a21e4ffaade33463d9f1ee9
Normal-2374.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2374.png,Normal/images/Normal-2374.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2374.png,Normal/masks/Normal-2374.png,True,False,299,299,L,146.14,271d3c9dcb984b2b60266d90d95d23f6
Normal-2375.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2375.png,Normal/images/Normal-2375.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2375.png,Normal/masks/Normal-2375.png,True,False,299,299,L,114.86,c33c4f5b582d634de2539fd05a7403b6
Normal-2376.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2376.png,Normal/images/Normal-2376.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2376.png,Normal/masks/Normal-2376.png,True,False,299,299,L,149.43,c717bc9def7dca4a128d291400339351
Normal-2377.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2377.png,Normal/images/Normal-2377.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2377.png,Normal/masks/Normal-2377.png,True,False,299,299,L,106.71,f6f0bdd87b8cdb15aa7dbcd2ff4cf666
Normal-2378.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2378.png,Normal/images/Normal-2378.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2378.png,Normal/masks/Normal-2378.png,True,False,299,299,L,120.93,6ec8ffbdd0da64e1e2a8d11b405c0d8a
Normal-2379.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2379.png,Normal/images/Normal-2379.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2379.png,Normal/masks/Normal-2379.png,True,False,299,299,L,111.03,9e836a5420f91b4b237d6354ab72284c
Normal-238.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-238.png,Normal/images/Normal-238.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-238.png,Normal/masks/Normal-238.png,True,False,299,299,L,108.35,002d6677c1d5fbbe25da940aec1d14c0
Normal-2380.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2380.png,Normal/images/Normal-2380.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2380.png,Normal/masks/Normal-2380.png,True,False,299,299,L,165.69,5ec441e9610a1df9123ce5786e97e9ca
Normal-2381.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2381.png,Normal/images/Normal-2381.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2381.png,Normal/masks/Normal-2381.png,True,False,299,299,L,123.67,553dabd984624b602dd7523eec03a969
Normal-2382.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2382.png,Normal/images/Normal-2382.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2382.png,Normal/masks/Normal-2382.png,True,False,299,299,L,117.03,b045146635a0968ea7caefc16738ba1e
Normal-2383.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2383.png,Normal/images/Normal-2383.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2383.png,Normal/masks/Normal-2383.png,True,False,299,299,L,112.16,4e184b990606c33081cdb93dc48faef4
Normal-2384.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2384.png,Normal/images/Normal-2384.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2384.png,Normal/masks/Normal-2384.png,True,False,299,299,L,127.48,b287d598bb0bc75cf437774e1cfe16d3
Normal-2385.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2385.png,Normal/images/Normal-2385.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2385.png,Normal/masks/Normal-2385.png,True,False,299,299,L,114.06,c7a82880577c5fd296b4bfc5a0dc7809
Normal-2386.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2386.png,Normal/images/Normal-2386.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2386.png,Normal/masks/Normal-2386.png,True,False,299,299,L,134.81,ab47c618fe7e551b0cc0a8c6968e9c36
Normal-2387.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2387.png,Normal/images/Normal-2387.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2387.png,Normal/masks/Normal-2387.png,True,False,299,299,L,132.76,21c1ca09d2b9d6ad8aff94b89efbd8dd
Normal-2388.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2388.png,Normal/images/Normal-2388.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2388.png,Normal/masks/Normal-2388.png,True,False,299,299,L,115.83,07faf9966385f60f8e08a9db2f53d4bb
Normal-2389.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2389.png,Normal/images/Normal-2389.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2389.png,Normal/masks/Normal-2389.png,True,False,299,299,L,116.21,d1834d59e8c5a7d342b58ea009d28180
Normal-239.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-239.png,Normal/images/Normal-239.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-239.png,Normal/masks/Normal-239.png,True,False,299,299,L,118.47,06e458b6b64d49b33efeacfa3ff6125b
Normal-2390.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2390.png,Normal/images/Normal-2390.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2390.png,Normal/masks/Normal-2390.png,True,False,299,299,L,110.01,e1e5bd654b3d3fea7d0bee080cc2165c
Normal-2391.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2391.png,Normal/images/Normal-2391.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2391.png,Normal/masks/Normal-2391.png,True,False,299,299,L,168.44,2d1717c937108ab151250807c2cb1b84
Normal-2392.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2392.png,Normal/images/Normal-2392.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2392.png,Normal/masks/Normal-2392.png,True,False,299,299,L,134.82,b89ac3adde2196e236ce1a066d0604c0
Normal-2393.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2393.png,Normal/images/Normal-2393.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2393.png,Normal/masks/Normal-2393.png,True,False,299,299,L,101.53,6c611d39807732d4a4480a6965d1b9cf
Normal-2394.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2394.png,Normal/images/Normal-2394.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2394.png,Normal/masks/Normal-2394.png,True,False,299,299,L,109.91,d52d0e08e8487135e2d73752d7d81bbb
Normal-2395.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2395.png,Normal/images/Normal-2395.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2395.png,Normal/masks/Normal-2395.png,True,False,299,299,L,122.23,86dc432c28374551bcf3246e1f4c01d5
Normal-2396.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2396.png,Normal/images/Normal-2396.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2396.png,Normal/masks/Normal-2396.png,True,False,299,299,L,178.73,72471988ae86138a4a5dd838db93e494
Normal-2397.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2397.png,Normal/images/Normal-2397.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2397.png,Normal/masks/Normal-2397.png,True,False,299,299,L,122.06,651e52690776c7c8d7d75daa3c2cb9f4
Normal-2398.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2398.png,Normal/images/Normal-2398.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2398.png,Normal/masks/Normal-2398.png,True,False,299,299,L,134.81,ec715fdff172dcb4cc4493b4a5f87321
Normal-2399.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2399.png,Normal/images/Normal-2399.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2399.png,Normal/masks/Normal-2399.png,True,False,299,299,L,116.63,bcbf995cc3cb447b9685c66b2fbefbad
Normal-24.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-24.png,Normal/images/Normal-24.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-24.png,Normal/masks/Normal-24.png,True,False,299,299,L,103.95,7a8e2558e60df49eec2c14dc7b6fd398
Normal-240.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-240.png,Normal/images/Normal-240.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-240.png,Normal/masks/Normal-240.png,True,False,299,299,L,116.33,63a9533bd387bff4e8d6039156ee1f4e
Normal-2400.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2400.png,Normal/images/Normal-2400.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2400.png,Normal/masks/Normal-2400.png,True,False,299,299,L,130.54,00c5cd7b464859ceab6e67193efb904a
Normal-2401.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2401.png,Normal/images/Normal-2401.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2401.png,Normal/masks/Normal-2401.png,True,False,299,299,L,64.04,27c8f7bdca4ba61e8fd0eecb57622938
Normal-2402.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2402.png,Normal/images/Normal-2402.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2402.png,Normal/masks/Normal-2402.png,True,False,299,299,L,106.99,062c2b5a646aa0beb4fab532b588819a
Normal-2403.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2403.png,Normal/images/Normal-2403.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2403.png,Normal/masks/Normal-2403.png,True,False,299,299,L,175.15,2ed9e235d0b662961e9c7023082f41c9
Normal-2404.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2404.png,Normal/images/Normal-2404.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2404.png,Normal/masks/Normal-2404.png,True,False,299,299,L,131.42,508fe4fa441609884ba4f05597800d54
Normal-2405.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2405.png,Normal/images/Normal-2405.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2405.png,Normal/masks/Normal-2405.png,True,False,299,299,L,114.95,7e139d4305cf45798363adaf14e8813e
Normal-2406.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2406.png,Normal/images/Normal-2406.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2406.png,Normal/masks/Normal-2406.png,True,False,299,299,L,168.99,b10596b0acf32cc244221fd8fb582c5a
Normal-2407.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2407.png,Normal/images/Normal-2407.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2407.png,Normal/masks/Normal-2407.png,True,False,299,299,L,102.68,d06d94fe3647b70d4053630f62ae842e
Normal-2408.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2408.png,Normal/images/Normal-2408.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2408.png,Normal/masks/Normal-2408.png,True,False,299,299,L,105.41,d09282133f7108d60873d378c358ad8e
Normal-2409.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2409.png,Normal/images/Normal-2409.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2409.png,Normal/masks/Normal-2409.png,True,False,299,299,L,111.47,89cfd4053f1af1f5e1cc7f7a441bce2e
Normal-241.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-241.png,Normal/images/Normal-241.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-241.png,Normal/masks/Normal-241.png,True,False,299,299,L,121.18,2cec6e1028e5580e8b242809856b8441
Normal-2410.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2410.png,Normal/images/Normal-2410.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2410.png,Normal/masks/Normal-2410.png,True,False,299,299,L,127.29,b2b962f14b916a2fcebba2b54ae1bd61
Normal-2411.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2411.png,Normal/images/Normal-2411.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2411.png,Normal/masks/Normal-2411.png,True,False,299,299,L,139.22,7723e2225c2d1a534a1220ca1780a3e6
Normal-2412.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2412.png,Normal/images/Normal-2412.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2412.png,Normal/masks/Normal-2412.png,True,False,299,299,L,134.18,87b3f5b4b14981f7eaf4b1928c791a17
Normal-2413.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2413.png,Normal/images/Normal-2413.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2413.png,Normal/masks/Normal-2413.png,True,False,299,299,L,125.49,7049a26cea7afa8665e3aec87a47a7a5
Normal-2414.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2414.png,Normal/images/Normal-2414.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2414.png,Normal/masks/Normal-2414.png,True,False,299,299,L,121.67,eacff90c79c14c78bdd7829aac4f1c82
Normal-2415.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2415.png,Normal/images/Normal-2415.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2415.png,Normal/masks/Normal-2415.png,True,False,299,299,L,100.68,fc2e547c1a14f0918e8c95f7c4b126c3
Normal-2416.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2416.png,Normal/images/Normal-2416.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2416.png,Normal/masks/Normal-2416.png,True,False,299,299,L,123.06,e00f7870e036b372cde1fe0a814869f4
Normal-2417.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2417.png,Normal/images/Normal-2417.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2417.png,Normal/masks/Normal-2417.png,True,False,299,299,L,136.43,87877f4cd784455f706ec6c4f08e2c7a
Normal-2418.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2418.png,Normal/images/Normal-2418.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2418.png,Normal/masks/Normal-2418.png,True,False,299,299,L,172.89,5a24f65203daf521d7619bd99958cbba
Normal-2419.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2419.png,Normal/images/Normal-2419.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2419.png,Normal/masks/Normal-2419.png,True,False,299,299,L,126.02,423a4c51a8df2ba9a12b590016aeccdc
Normal-242.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-242.png,Normal/images/Normal-242.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-242.png,Normal/masks/Normal-242.png,True,False,299,299,L,122.74,28575980c7586d34a1b7d61f2f31702d
Normal-2420.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2420.png,Normal/images/Normal-2420.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2420.png,Normal/masks/Normal-2420.png,True,False,299,299,L,97.05,8b927e010956bce3660005b073ab51d5
Normal-2421.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2421.png,Normal/images/Normal-2421.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2421.png,Normal/masks/Normal-2421.png,True,False,299,299,L,113.27,33bc27443994922bde49970b31fbaa52
Normal-2422.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2422.png,Normal/images/Normal-2422.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2422.png,Normal/masks/Normal-2422.png,True,False,299,299,L,108.21,c091182035d17d944b7a496b3b973003
Normal-2423.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2423.png,Normal/images/Normal-2423.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2423.png,Normal/masks/Normal-2423.png,True,False,299,299,L,176.9,44efbd4c1330cbfa85c02e3a881b07b1
Normal-2424.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2424.png,Normal/images/Normal-2424.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2424.png,Normal/masks/Normal-2424.png,True,False,299,299,L,109.42,f9a464eaf838f58e1872b1819ed27664
Normal-2425.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2425.png,Normal/images/Normal-2425.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2425.png,Normal/masks/Normal-2425.png,True,False,299,299,L,120.53,d8ed6ace01c073bc671ccba8ce4fefd0
Normal-2426.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2426.png,Normal/images/Normal-2426.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2426.png,Normal/masks/Normal-2426.png,True,False,299,299,L,114.95,b074170735459c8620778f1b32ba6b0a
Normal-2427.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2427.png,Normal/images/Normal-2427.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2427.png,Normal/masks/Normal-2427.png,True,False,299,299,L,161.26,dda941fd70d3d9c489a1546b2354058a
Normal-2428.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2428.png,Normal/images/Normal-2428.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2428.png,Normal/masks/Normal-2428.png,True,False,299,299,L,108.15,0e7df4b5abcc6f12e095bcabbc697950
Normal-2429.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2429.png,Normal/images/Normal-2429.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2429.png,Normal/masks/Normal-2429.png,True,False,299,299,L,151.05,1a0d5a5d3a044555f56c2ca565779079
Normal-243.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-243.png,Normal/images/Normal-243.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-243.png,Normal/masks/Normal-243.png,True,False,299,299,L,135.1,8bea429db6a3cc3164430cba1ab25a77
Normal-2430.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2430.png,Normal/images/Normal-2430.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2430.png,Normal/masks/Normal-2430.png,True,False,299,299,L,127.18,0a38d77a84fca505a01fc035c9c8ca24
Normal-2431.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2431.png,Normal/images/Normal-2431.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2431.png,Normal/masks/Normal-2431.png,True,False,299,299,L,130.71,d096c9605b6c15a2ea871ac578c0952b
Normal-2432.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2432.png,Normal/images/Normal-2432.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2432.png,Normal/masks/Normal-2432.png,True,False,299,299,L,167.47,4018373cba7148b4b868ee5eb5789a36
Normal-2433.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2433.png,Normal/images/Normal-2433.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2433.png,Normal/masks/Normal-2433.png,True,False,299,299,L,108.76,a0033f01523a976adb7d7912190f2a06
Normal-2434.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2434.png,Normal/images/Normal-2434.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2434.png,Normal/masks/Normal-2434.png,True,False,299,299,L,185.03,811629dc839b8b764890ca88a63d928d
Normal-2435.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2435.png,Normal/images/Normal-2435.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2435.png,Normal/masks/Normal-2435.png,True,False,299,299,L,160.46,39852b2ec62b8adcc3012788962fb1c3
Normal-2436.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2436.png,Normal/images/Normal-2436.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2436.png,Normal/masks/Normal-2436.png,True,False,299,299,L,145.6,858430a5bd6683b564620f5362e2564a
Normal-2437.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2437.png,Normal/images/Normal-2437.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2437.png,Normal/masks/Normal-2437.png,True,False,299,299,L,159.11,f8719e60bf8cbed618dc4aa89cb64621
Normal-2438.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2438.png,Normal/images/Normal-2438.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2438.png,Normal/masks/Normal-2438.png,True,False,299,299,L,116.75,d90f5c2c0f96976881590f29694c8f71
Normal-2439.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2439.png,Normal/images/Normal-2439.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2439.png,Normal/masks/Normal-2439.png,True,False,299,299,L,149.33,44737d31eb067fba49fe1c87cb9cdd36
Normal-244.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-244.png,Normal/images/Normal-244.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-244.png,Normal/masks/Normal-244.png,True,False,299,299,L,124.98,005b203df11b12f4910e6045bfdcf8ac
Normal-2440.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2440.png,Normal/images/Normal-2440.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2440.png,Normal/masks/Normal-2440.png,True,False,299,299,L,111.46,1be478bcce57c7d0948d61fefd8e3b13
Normal-2441.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2441.png,Normal/images/Normal-2441.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2441.png,Normal/masks/Normal-2441.png,True,False,299,299,L,144.69,5497570e8713e9b684d4a708c5e2cd5b
Normal-2442.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2442.png,Normal/images/Normal-2442.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2442.png,Normal/masks/Normal-2442.png,True,False,299,299,L,129.68,2b3336475d95addd52a87a789287728e
Normal-2443.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2443.png,Normal/images/Normal-2443.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2443.png,Normal/masks/Normal-2443.png,True,False,299,299,L,119.59,a98259f2f0a29af9814590df6c801c86
Normal-2444.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2444.png,Normal/images/Normal-2444.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2444.png,Normal/masks/Normal-2444.png,True,False,299,299,L,118.56,db5695d14321bd526f2ea7a0fb2c96c9
Normal-2445.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2445.png,Normal/images/Normal-2445.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2445.png,Normal/masks/Normal-2445.png,True,False,299,299,L,111.48,78eaf43ddc3978130fde8db375fcc0b3
Normal-2446.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2446.png,Normal/images/Normal-2446.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2446.png,Normal/masks/Normal-2446.png,True,False,299,299,L,141.77,f54c03f56f25bead7d49ca9b6da73ca2
Normal-2447.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2447.png,Normal/images/Normal-2447.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2447.png,Normal/masks/Normal-2447.png,True,False,299,299,L,133.65,1bbfb14d6f44d942fc6e74754d5da279
Normal-2448.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2448.png,Normal/images/Normal-2448.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2448.png,Normal/masks/Normal-2448.png,True,False,299,299,L,114.23,e3b3252d9c540d130090a0780fea1d52
Normal-2449.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2449.png,Normal/images/Normal-2449.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2449.png,Normal/masks/Normal-2449.png,True,False,299,299,L,134.57,c6f55bc68c2d87efd9b6c251c3ac1776
Normal-245.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-245.png,Normal/images/Normal-245.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-245.png,Normal/masks/Normal-245.png,True,False,299,299,L,138.97,179cb178383006492ab68d0c18149f58
Normal-2450.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2450.png,Normal/images/Normal-2450.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2450.png,Normal/masks/Normal-2450.png,True,False,299,299,L,154.19,1dc42df30d43721f245f8ccf7a5d6f10
Normal-2451.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2451.png,Normal/images/Normal-2451.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2451.png,Normal/masks/Normal-2451.png,True,False,299,299,L,89.47,e84e55b62a61f4fe1cc1b259fb20cbfe
Normal-2452.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2452.png,Normal/images/Normal-2452.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2452.png,Normal/masks/Normal-2452.png,True,False,299,299,L,143.97,050f7fe242ff57a8af0d88a7dd73075d
Normal-2453.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2453.png,Normal/images/Normal-2453.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2453.png,Normal/masks/Normal-2453.png,True,False,299,299,L,120.45,ab910d739fc5a24b0b8b92cbd92f0db5
Normal-2454.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2454.png,Normal/images/Normal-2454.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2454.png,Normal/masks/Normal-2454.png,True,False,299,299,L,170.9,41975061aa0377aa5a3bf24adaea4e10
Normal-2455.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2455.png,Normal/images/Normal-2455.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2455.png,Normal/masks/Normal-2455.png,True,False,299,299,L,117.83,e7dc630f2b80001ff50d3db99462cdbd
Normal-2456.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2456.png,Normal/images/Normal-2456.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2456.png,Normal/masks/Normal-2456.png,True,False,299,299,L,169.18,cf790c93182e7299095bb8dc71462420
Normal-2457.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2457.png,Normal/images/Normal-2457.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2457.png,Normal/masks/Normal-2457.png,True,False,299,299,L,103.64,b245d1627d5747fb1b6a8ce0b4fec85c
Normal-2458.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2458.png,Normal/images/Normal-2458.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2458.png,Normal/masks/Normal-2458.png,True,False,299,299,L,171.11,e0b7493a9e90ec437389a259d79499e0
Normal-2459.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2459.png,Normal/images/Normal-2459.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2459.png,Normal/masks/Normal-2459.png,True,False,299,299,L,113.41,993c28c98371d05fb9addbeb4e328c7f
Normal-246.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-246.png,Normal/images/Normal-246.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-246.png,Normal/masks/Normal-246.png,True,False,299,299,L,129.62,76b4b147a2461e52a006cb2e9ea0f1c6
Normal-2460.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2460.png,Normal/images/Normal-2460.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2460.png,Normal/masks/Normal-2460.png,True,False,299,299,L,153.13,1a9dab38f8e83d2ce0b1238f818dbe5f
Normal-2461.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2461.png,Normal/images/Normal-2461.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2461.png,Normal/masks/Normal-2461.png,True,False,299,299,L,138.96,d2484633bf5bef9e9db3e6873ec234c8
Normal-2462.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2462.png,Normal/images/Normal-2462.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2462.png,Normal/masks/Normal-2462.png,True,False,299,299,L,162.74,e1e2bc5d7576304e393aedb339dc3f43
Normal-2463.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2463.png,Normal/images/Normal-2463.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2463.png,Normal/masks/Normal-2463.png,True,False,299,299,L,120.29,6f351d4dbea421d56b16f9d9cb58adc2
Normal-2464.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2464.png,Normal/images/Normal-2464.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2464.png,Normal/masks/Normal-2464.png,True,False,299,299,L,143.42,0870531bef19dd5380a64550d1590c9f
Normal-2465.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2465.png,Normal/images/Normal-2465.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2465.png,Normal/masks/Normal-2465.png,True,False,299,299,L,119.48,819608c1a510787c628ca233ba39a833
Normal-2466.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2466.png,Normal/images/Normal-2466.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2466.png,Normal/masks/Normal-2466.png,True,False,299,299,L,111.07,6f32152a838c6a39f1eae535ac8e0eae
Normal-2467.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2467.png,Normal/images/Normal-2467.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2467.png,Normal/masks/Normal-2467.png,True,False,299,299,L,159.33,e2c910098cb2c095917ac25dbe77ac6a
Normal-2468.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2468.png,Normal/images/Normal-2468.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2468.png,Normal/masks/Normal-2468.png,True,False,299,299,L,166.31,93bd94109a8c68bd62ba19462cff9a71
Normal-2469.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2469.png,Normal/images/Normal-2469.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2469.png,Normal/masks/Normal-2469.png,True,False,299,299,L,114.94,360e57007e07e443deb78e027d9c5af2
Normal-247.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-247.png,Normal/images/Normal-247.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-247.png,Normal/masks/Normal-247.png,True,False,299,299,L,153.83,15f5b9fee959aa32512cf917d9a68dba
Normal-2470.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2470.png,Normal/images/Normal-2470.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2470.png,Normal/masks/Normal-2470.png,True,False,299,299,L,150.43,631c0ed4c9867dc60b9422afd657ad21
Normal-2471.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2471.png,Normal/images/Normal-2471.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2471.png,Normal/masks/Normal-2471.png,True,False,299,299,L,112.13,ab75054cac5b20dcff784a2c065e7b25
Normal-2472.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2472.png,Normal/images/Normal-2472.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2472.png,Normal/masks/Normal-2472.png,True,False,299,299,L,106.45,ea5d05aae804cfe888f962d32ba5b6c0
Normal-2473.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2473.png,Normal/images/Normal-2473.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2473.png,Normal/masks/Normal-2473.png,True,False,299,299,L,119.01,6ac88f70e1057578b0de192671bae7e7
Normal-2474.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2474.png,Normal/images/Normal-2474.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2474.png,Normal/masks/Normal-2474.png,True,False,299,299,L,152.69,178712f9bd600de6c367e933de9cd67c
Normal-2475.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2475.png,Normal/images/Normal-2475.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2475.png,Normal/masks/Normal-2475.png,True,False,299,299,L,108.2,1615b0f9a0a85533e152009397baaae0
Normal-2476.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2476.png,Normal/images/Normal-2476.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2476.png,Normal/masks/Normal-2476.png,True,False,299,299,L,79.74,54e08e875fc33627c70e31258a918d1c
Normal-2477.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2477.png,Normal/images/Normal-2477.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2477.png,Normal/masks/Normal-2477.png,True,False,299,299,L,133.36,5d0cdeb3e467551fb1eb5b89cb25a169
Normal-2478.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2478.png,Normal/images/Normal-2478.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2478.png,Normal/masks/Normal-2478.png,True,False,299,299,L,130.38,6b31b3e5436dc9e21d2b7d7ee39d7bb3
Normal-2479.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2479.png,Normal/images/Normal-2479.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2479.png,Normal/masks/Normal-2479.png,True,False,299,299,L,128.75,dca3a198d60439908512e8a34e48967a
Normal-248.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-248.png,Normal/images/Normal-248.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-248.png,Normal/masks/Normal-248.png,True,False,299,299,L,113.21,ec7421e5586bb85dc6f45f19ffbf55a6
Normal-2480.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2480.png,Normal/images/Normal-2480.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2480.png,Normal/masks/Normal-2480.png,True,False,299,299,L,128.82,033ca28805e5fa612da7ced4f1ae3016
Normal-2481.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2481.png,Normal/images/Normal-2481.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2481.png,Normal/masks/Normal-2481.png,True,False,299,299,L,176.38,ec5abfd4ceb41e2dbe90406409d495da
Normal-2482.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2482.png,Normal/images/Normal-2482.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2482.png,Normal/masks/Normal-2482.png,True,False,299,299,L,125.78,dd6faed8ce77f17ecb8d6ec0937643a6
Normal-2483.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2483.png,Normal/images/Normal-2483.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2483.png,Normal/masks/Normal-2483.png,True,False,299,299,L,169.89,19324c62514d2013d5cbc7b0378d260e
Normal-2484.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2484.png,Normal/images/Normal-2484.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2484.png,Normal/masks/Normal-2484.png,True,False,299,299,L,110.19,2cb1648f43a34defad51c8ac0ca7c2d2
Normal-2485.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2485.png,Normal/images/Normal-2485.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2485.png,Normal/masks/Normal-2485.png,True,False,299,299,L,143.65,e14747e8f9d70007e7efb1f8968b7e4a
Normal-2486.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2486.png,Normal/images/Normal-2486.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2486.png,Normal/masks/Normal-2486.png,True,False,299,299,L,111.24,57172862c2b0a1517e3dc2dbd1a099ac
Normal-2487.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2487.png,Normal/images/Normal-2487.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2487.png,Normal/masks/Normal-2487.png,True,False,299,299,L,139.3,4d516a32f5e71a2c85165dd25a2d3081
Normal-2488.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2488.png,Normal/images/Normal-2488.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2488.png,Normal/masks/Normal-2488.png,True,False,299,299,L,120.91,1d6eda6dd7e6f73b5cf9617cf7d36b0f
Normal-2489.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2489.png,Normal/images/Normal-2489.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2489.png,Normal/masks/Normal-2489.png,True,False,299,299,L,124.99,cf84316bd3eef7075355307ceabc6870
Normal-249.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-249.png,Normal/images/Normal-249.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-249.png,Normal/masks/Normal-249.png,True,False,299,299,L,135.28,f77f211260f99bc2abb5e70a8e43d56c
Normal-2490.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2490.png,Normal/images/Normal-2490.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2490.png,Normal/masks/Normal-2490.png,True,False,299,299,L,171.99,1ce6c94b587d9a7cd35f81079a375189
Normal-2491.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2491.png,Normal/images/Normal-2491.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2491.png,Normal/masks/Normal-2491.png,True,False,299,299,L,137.4,b8e6cc223e8b7a34a415a2de3bcef531
Normal-2492.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2492.png,Normal/images/Normal-2492.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2492.png,Normal/masks/Normal-2492.png,True,False,299,299,L,121.21,22809fd14c5e79d15d9fdcbde14de192
Normal-2493.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2493.png,Normal/images/Normal-2493.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2493.png,Normal/masks/Normal-2493.png,True,False,299,299,L,144.45,dea65ebd8a0e171b6713da7a7b170840
Normal-2494.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2494.png,Normal/images/Normal-2494.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2494.png,Normal/masks/Normal-2494.png,True,False,299,299,L,131.18,a1ca7ccb3dc6d64b79934bf0e2a9d451
Normal-2495.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2495.png,Normal/images/Normal-2495.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2495.png,Normal/masks/Normal-2495.png,True,False,299,299,L,134.0,d08a9cfc60470ff0f7e275bd257eee4a
Normal-2496.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2496.png,Normal/images/Normal-2496.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2496.png,Normal/masks/Normal-2496.png,True,False,299,299,L,130.16,c9e141faef0dda932aae58126541818a
Normal-2497.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2497.png,Normal/images/Normal-2497.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2497.png,Normal/masks/Normal-2497.png,True,False,299,299,L,168.65,5ce185619edae4535e39f33e85b31491
Normal-2498.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2498.png,Normal/images/Normal-2498.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2498.png,Normal/masks/Normal-2498.png,True,False,299,299,L,169.99,c82abb4f7942516fec1d47f0268f2846
Normal-2499.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2499.png,Normal/images/Normal-2499.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2499.png,Normal/masks/Normal-2499.png,True,False,299,299,L,89.31,398a9b67b64fa83b821be3fd83042f6f
Normal-25.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-25.png,Normal/images/Normal-25.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-25.png,Normal/masks/Normal-25.png,True,False,299,299,L,138.02,8971919170b3d1293147c06469d67799
Normal-250.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-250.png,Normal/images/Normal-250.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-250.png,Normal/masks/Normal-250.png,True,False,299,299,L,115.09,a5c0dcc3e143b101b979f16ab9630d9e
Normal-2500.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2500.png,Normal/images/Normal-2500.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2500.png,Normal/masks/Normal-2500.png,True,False,299,299,L,110.76,d0ca59227b34dbe9d8d5eb6ec221be55
Normal-2501.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2501.png,Normal/images/Normal-2501.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2501.png,Normal/masks/Normal-2501.png,True,False,299,299,L,107.3,b85685a1b9d0c60626a06e3ea3ffbfad
Normal-2502.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2502.png,Normal/images/Normal-2502.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2502.png,Normal/masks/Normal-2502.png,True,False,299,299,L,127.07,2342e058b884875b72f8c7e6f9a54ec6
Normal-2503.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2503.png,Normal/images/Normal-2503.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2503.png,Normal/masks/Normal-2503.png,True,False,299,299,L,129.79,7d097ed47a0dd22c4b028f6996a02e0c
Normal-2504.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2504.png,Normal/images/Normal-2504.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2504.png,Normal/masks/Normal-2504.png,True,False,299,299,L,151.03,431d1c29681fc1edbc46f66b34cb4578
Normal-2505.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2505.png,Normal/images/Normal-2505.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2505.png,Normal/masks/Normal-2505.png,True,False,299,299,L,163.2,c3a4d96ca90626d507cfc2167ba9b18a
Normal-2506.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2506.png,Normal/images/Normal-2506.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2506.png,Normal/masks/Normal-2506.png,True,False,299,299,L,159.56,3ac867dabb4eeacbea56b7c261a42274
Normal-2507.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2507.png,Normal/images/Normal-2507.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2507.png,Normal/masks/Normal-2507.png,True,False,299,299,L,121.16,9e9849c27605ffd33f84d4e798370ca1
Normal-2508.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2508.png,Normal/images/Normal-2508.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2508.png,Normal/masks/Normal-2508.png,True,False,299,299,L,133.43,150d5497dbab5dc7f37571849fee3879
Normal-2509.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2509.png,Normal/images/Normal-2509.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2509.png,Normal/masks/Normal-2509.png,True,False,299,299,L,102.36,e176099059fd2c630786dc7746c294c5
Normal-251.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-251.png,Normal/images/Normal-251.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-251.png,Normal/masks/Normal-251.png,True,False,299,299,L,119.08,cc9af14ecbfab24ef008058e59642adc
Normal-2510.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2510.png,Normal/images/Normal-2510.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2510.png,Normal/masks/Normal-2510.png,True,False,299,299,L,125.67,7b7cdaf93fb52b19be91f33aaa9658be
Normal-2511.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2511.png,Normal/images/Normal-2511.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2511.png,Normal/masks/Normal-2511.png,True,False,299,299,L,111.08,a276ed9440c1bf5ca43c303310133f07
Normal-2512.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2512.png,Normal/images/Normal-2512.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2512.png,Normal/masks/Normal-2512.png,True,False,299,299,L,97.76,cfb2630715b89def893b32d666565bfc
Normal-2513.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2513.png,Normal/images/Normal-2513.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2513.png,Normal/masks/Normal-2513.png,True,False,299,299,L,98.46,5415e54b6dfd4936485114bb6574491c
Normal-2514.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2514.png,Normal/images/Normal-2514.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2514.png,Normal/masks/Normal-2514.png,True,False,299,299,L,140.48,09a514e339b9c319998aaf161312ea81
Normal-2515.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2515.png,Normal/images/Normal-2515.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2515.png,Normal/masks/Normal-2515.png,True,False,299,299,L,106.49,37ef1f05fe37ad5d8a9efea600a4eb59
Normal-2516.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2516.png,Normal/images/Normal-2516.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2516.png,Normal/masks/Normal-2516.png,True,False,299,299,L,118.78,cde5d71872da83f3a0b6aec91a0ea68f
Normal-2517.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2517.png,Normal/images/Normal-2517.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2517.png,Normal/masks/Normal-2517.png,True,False,299,299,L,116.38,c4f29a478b2b505ed04b1675f7fe0fe3
Normal-2518.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2518.png,Normal/images/Normal-2518.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2518.png,Normal/masks/Normal-2518.png,True,False,299,299,L,156.57,080e2786ce6d27626f52ffb5896256d6
Normal-2519.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2519.png,Normal/images/Normal-2519.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2519.png,Normal/masks/Normal-2519.png,True,False,299,299,L,114.49,ae8c997349c428c51678870cf235e1c5
Normal-252.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-252.png,Normal/images/Normal-252.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-252.png,Normal/masks/Normal-252.png,True,False,299,299,L,106.1,dd79409b456d1f3486f8895497de8891
Normal-2520.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2520.png,Normal/images/Normal-2520.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2520.png,Normal/masks/Normal-2520.png,True,False,299,299,L,125.11,335e4ee6153f42b6cd0f4a3ffebe0968
Normal-2521.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2521.png,Normal/images/Normal-2521.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2521.png,Normal/masks/Normal-2521.png,True,False,299,299,L,153.54,14432ac06584c5607424d799d17ff0fd
Normal-2522.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2522.png,Normal/images/Normal-2522.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2522.png,Normal/masks/Normal-2522.png,True,False,299,299,L,118.83,5607212dc1e30ea8b6b1bae5f3b7e870
Normal-2523.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2523.png,Normal/images/Normal-2523.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2523.png,Normal/masks/Normal-2523.png,True,False,299,299,L,114.96,c18564bd957bf5ab3179f61d3c294f38
Normal-2524.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2524.png,Normal/images/Normal-2524.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2524.png,Normal/masks/Normal-2524.png,True,False,299,299,L,151.0,69ecc361c236819aa2e342339e89bc63
Normal-2525.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2525.png,Normal/images/Normal-2525.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2525.png,Normal/masks/Normal-2525.png,True,False,299,299,L,157.11,e99498d54c82d48437124e6f5d35294e
Normal-2526.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2526.png,Normal/images/Normal-2526.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2526.png,Normal/masks/Normal-2526.png,True,False,299,299,L,129.6,360a77b5e1b7f567752737993ccd31a7
Normal-2527.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2527.png,Normal/images/Normal-2527.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2527.png,Normal/masks/Normal-2527.png,True,False,299,299,L,158.38,6958710a56f589ba24f1c022983cc04d
Normal-2528.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2528.png,Normal/images/Normal-2528.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2528.png,Normal/masks/Normal-2528.png,True,False,299,299,L,145.94,457c9885e886446c94be7cdfdc6845e1
Normal-2529.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2529.png,Normal/images/Normal-2529.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2529.png,Normal/masks/Normal-2529.png,True,False,299,299,L,103.4,630d05b23bd821d63c0aab343c455d01
Normal-253.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-253.png,Normal/images/Normal-253.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-253.png,Normal/masks/Normal-253.png,True,False,299,299,L,128.76,fe92f8a22e8add05121dfb1bed88e0fc
Normal-2530.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2530.png,Normal/images/Normal-2530.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2530.png,Normal/masks/Normal-2530.png,True,False,299,299,L,146.22,3d0aceab65e825834a96d88379acb309
Normal-2531.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2531.png,Normal/images/Normal-2531.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2531.png,Normal/masks/Normal-2531.png,True,False,299,299,L,128.15,375d12fc1138b2a5627812d0da79adca
Normal-2532.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2532.png,Normal/images/Normal-2532.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2532.png,Normal/masks/Normal-2532.png,True,False,299,299,L,109.86,f6d2d9c928c2675425bafa726fbed8d9
Normal-2533.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2533.png,Normal/images/Normal-2533.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2533.png,Normal/masks/Normal-2533.png,True,False,299,299,L,136.09,beff733814d5c7e1d6eaf122eee055e0
Normal-2534.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2534.png,Normal/images/Normal-2534.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2534.png,Normal/masks/Normal-2534.png,True,False,299,299,L,168.18,6db90a1850e8449f01e1221dfedcdd39
Normal-2535.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2535.png,Normal/images/Normal-2535.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2535.png,Normal/masks/Normal-2535.png,True,False,299,299,L,172.28,52bd71cafe044b9d549acf3149b773fc
Normal-2536.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2536.png,Normal/images/Normal-2536.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2536.png,Normal/masks/Normal-2536.png,True,False,299,299,L,159.02,4fb029e7cd8fd761e4aff7ff3ff73520
Normal-2537.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2537.png,Normal/images/Normal-2537.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2537.png,Normal/masks/Normal-2537.png,True,False,299,299,L,159.97,97b0765bea9e95ec4916ebaab079376c
Normal-2538.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2538.png,Normal/images/Normal-2538.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2538.png,Normal/masks/Normal-2538.png,True,False,299,299,L,132.12,d55fcab09691653bf865fbf632c7d1bc
Normal-2539.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2539.png,Normal/images/Normal-2539.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2539.png,Normal/masks/Normal-2539.png,True,False,299,299,L,120.24,5b6124479db2d8787a1168ba583c3c7b
Normal-254.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-254.png,Normal/images/Normal-254.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-254.png,Normal/masks/Normal-254.png,True,False,299,299,L,131.4,81be9dbad51648a6ab489cb8c84e6f5e
Normal-2540.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2540.png,Normal/images/Normal-2540.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2540.png,Normal/masks/Normal-2540.png,True,False,299,299,L,176.14,ea54165bdf283445735ca5a0abf7d1f4
Normal-2541.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2541.png,Normal/images/Normal-2541.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2541.png,Normal/masks/Normal-2541.png,True,False,299,299,L,151.88,9f0959c87a9dbcf12af59d511e8b1d18
Normal-2542.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2542.png,Normal/images/Normal-2542.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2542.png,Normal/masks/Normal-2542.png,True,False,299,299,L,127.5,760131ff44bcd2df28445ccbf9487683
Normal-2543.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2543.png,Normal/images/Normal-2543.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2543.png,Normal/masks/Normal-2543.png,True,False,299,299,L,132.26,ac3f8671209a62f342aabeb8b7353f62
Normal-2544.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2544.png,Normal/images/Normal-2544.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2544.png,Normal/masks/Normal-2544.png,True,False,299,299,L,112.29,390683c05e4d930f8dc61eaebb49835f
Normal-2545.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2545.png,Normal/images/Normal-2545.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2545.png,Normal/masks/Normal-2545.png,True,False,299,299,L,134.09,56d1cfbf88078d522459fe9092ac9743
Normal-2546.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2546.png,Normal/images/Normal-2546.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2546.png,Normal/masks/Normal-2546.png,True,False,299,299,L,115.75,42f7a390dcc3f645521b6410a920a086
Normal-2547.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2547.png,Normal/images/Normal-2547.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2547.png,Normal/masks/Normal-2547.png,True,False,299,299,L,94.5,3eff04c8a4b0873d2f20cd082730e81f
Normal-2548.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2548.png,Normal/images/Normal-2548.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2548.png,Normal/masks/Normal-2548.png,True,False,299,299,L,139.89,7fecf814bb143cfe52e4b1777f047069
Normal-2549.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2549.png,Normal/images/Normal-2549.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2549.png,Normal/masks/Normal-2549.png,True,False,299,299,L,184.51,1a798a66964506fba10d86f6130272f1
Normal-255.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-255.png,Normal/images/Normal-255.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-255.png,Normal/masks/Normal-255.png,True,False,299,299,L,121.39,6bb52c2c0f66bc3887755666a06e45b9
Normal-2550.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2550.png,Normal/images/Normal-2550.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2550.png,Normal/masks/Normal-2550.png,True,False,299,299,L,106.52,c27206cba5372db42dda7dce52279e3e
Normal-2551.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2551.png,Normal/images/Normal-2551.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2551.png,Normal/masks/Normal-2551.png,True,False,299,299,L,133.12,40cdfff470915edd8bd957eceb565b71
Normal-2552.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2552.png,Normal/images/Normal-2552.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2552.png,Normal/masks/Normal-2552.png,True,False,299,299,L,152.17,328b7ca11c1c2c3042344483493b98d4
Normal-2553.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2553.png,Normal/images/Normal-2553.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2553.png,Normal/masks/Normal-2553.png,True,False,299,299,L,109.48,a9d9a5bb1be44fdf99832247d7c525e5
Normal-2554.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2554.png,Normal/images/Normal-2554.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2554.png,Normal/masks/Normal-2554.png,True,False,299,299,L,104.34,8cf93739d5b08276055145cccbcbbb3f
Normal-2555.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2555.png,Normal/images/Normal-2555.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2555.png,Normal/masks/Normal-2555.png,True,False,299,299,L,149.43,5c6f536c03cc14e3114202a5c66731b1
Normal-2556.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2556.png,Normal/images/Normal-2556.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2556.png,Normal/masks/Normal-2556.png,True,False,299,299,L,139.46,62a2f557b16d30fa93e91db8148ca077
Normal-2557.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2557.png,Normal/images/Normal-2557.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2557.png,Normal/masks/Normal-2557.png,True,False,299,299,L,139.9,3da4c2630470f1016e3318ba0f69eca0
Normal-2558.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2558.png,Normal/images/Normal-2558.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2558.png,Normal/masks/Normal-2558.png,True,False,299,299,L,112.03,a0c7f4921b2004aae41d356061ba15c8
Normal-2559.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2559.png,Normal/images/Normal-2559.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2559.png,Normal/masks/Normal-2559.png,True,False,299,299,L,177.89,b2956519971423de32f80e88c54d7796
Normal-256.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-256.png,Normal/images/Normal-256.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-256.png,Normal/masks/Normal-256.png,True,False,299,299,L,117.98,f49c5627caba07e2d754b686d2b68ca2
Normal-2560.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2560.png,Normal/images/Normal-2560.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2560.png,Normal/masks/Normal-2560.png,True,False,299,299,L,107.2,4c3c9c2bb6dced21652c69c8ece12398
Normal-2561.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2561.png,Normal/images/Normal-2561.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2561.png,Normal/masks/Normal-2561.png,True,False,299,299,L,144.28,cd3d7ef3c75921623d5a23e23d191dee
Normal-2562.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2562.png,Normal/images/Normal-2562.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2562.png,Normal/masks/Normal-2562.png,True,False,299,299,L,160.19,6c804ad1d80091bf8e5cae80ad5efbe1
Normal-2563.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2563.png,Normal/images/Normal-2563.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2563.png,Normal/masks/Normal-2563.png,True,False,299,299,L,111.19,cb3ac0b4c300cfd0415fcc46c7fb8405
Normal-2564.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2564.png,Normal/images/Normal-2564.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2564.png,Normal/masks/Normal-2564.png,True,False,299,299,L,111.38,08ccfe50b86dc63f3b003c23f74c3d21
Normal-2565.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2565.png,Normal/images/Normal-2565.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2565.png,Normal/masks/Normal-2565.png,True,False,299,299,L,116.37,7a34a55774f0b312c43f098d2fcb8c4b
Normal-2566.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2566.png,Normal/images/Normal-2566.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2566.png,Normal/masks/Normal-2566.png,True,False,299,299,L,117.77,23d3bf69469b034749710de844c6e6f5
Normal-2567.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2567.png,Normal/images/Normal-2567.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2567.png,Normal/masks/Normal-2567.png,True,False,299,299,L,112.41,a3d1080cc8124b6b415319fe977f0d36
Normal-2568.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2568.png,Normal/images/Normal-2568.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2568.png,Normal/masks/Normal-2568.png,True,False,299,299,L,169.53,6244c3ea445d3d5c3f6bedb2e8765aed
Normal-2569.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2569.png,Normal/images/Normal-2569.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2569.png,Normal/masks/Normal-2569.png,True,False,299,299,L,164.7,97ea9599c700a3e7f364a2234c86514c
Normal-257.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-257.png,Normal/images/Normal-257.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-257.png,Normal/masks/Normal-257.png,True,False,299,299,L,120.17,deab3b0ef9693ae7724821f76947c11c
Normal-2570.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2570.png,Normal/images/Normal-2570.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2570.png,Normal/masks/Normal-2570.png,True,False,299,299,L,152.52,c96ab8e06c5027b56e40eb4a6018c4af
Normal-2571.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2571.png,Normal/images/Normal-2571.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2571.png,Normal/masks/Normal-2571.png,True,False,299,299,L,169.23,4a23d4183064a52bf5c8fd0201226b93
Normal-2572.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2572.png,Normal/images/Normal-2572.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2572.png,Normal/masks/Normal-2572.png,True,False,299,299,L,145.68,32cba44d33a67fd5a654d136ed6c1e95
Normal-2573.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2573.png,Normal/images/Normal-2573.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2573.png,Normal/masks/Normal-2573.png,True,False,299,299,L,112.86,718bbd01552f129556bc544fd2afabe4
Normal-2574.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2574.png,Normal/images/Normal-2574.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2574.png,Normal/masks/Normal-2574.png,True,False,299,299,L,168.01,a20a2a1c8cc569f40bd3af677280ec7c
Normal-2575.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2575.png,Normal/images/Normal-2575.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2575.png,Normal/masks/Normal-2575.png,True,False,299,299,L,131.35,9a3cf3dd8d9b72a11b8a9861393e30a3
Normal-2576.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2576.png,Normal/images/Normal-2576.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2576.png,Normal/masks/Normal-2576.png,True,False,299,299,L,178.0,4dcad970b0f619e55f9fdbc8cb1302d6
Normal-2577.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2577.png,Normal/images/Normal-2577.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2577.png,Normal/masks/Normal-2577.png,True,False,299,299,L,138.94,ee8cb730591578dac1efb9b1e89422b1
Normal-2578.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2578.png,Normal/images/Normal-2578.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2578.png,Normal/masks/Normal-2578.png,True,False,299,299,L,157.79,d4e8ca4520ab22e8842b32b708b10be3
Normal-2579.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2579.png,Normal/images/Normal-2579.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2579.png,Normal/masks/Normal-2579.png,True,False,299,299,L,151.58,0093d3279c5cb84a55eb88ac796c36c0
Normal-258.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-258.png,Normal/images/Normal-258.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-258.png,Normal/masks/Normal-258.png,True,False,299,299,L,119.39,87a743e6208e8bb572d26b40ebbd8f49
Normal-2580.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2580.png,Normal/images/Normal-2580.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2580.png,Normal/masks/Normal-2580.png,True,False,299,299,L,178.91,7fd33b4bdf7f2b0d4a6cb6e32bb78eec
Normal-2581.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2581.png,Normal/images/Normal-2581.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2581.png,Normal/masks/Normal-2581.png,True,False,299,299,L,112.38,4a03ad276bbdf6c50bb9b28d74276132
Normal-2582.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2582.png,Normal/images/Normal-2582.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2582.png,Normal/masks/Normal-2582.png,True,False,299,299,L,108.27,c492eec8c9264b88dd1e2873dc1d7d1f
Normal-2583.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2583.png,Normal/images/Normal-2583.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2583.png,Normal/masks/Normal-2583.png,True,False,299,299,L,127.15,bda9aa1c2fa58ededdff3f9927827bec
Normal-2584.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2584.png,Normal/images/Normal-2584.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2584.png,Normal/masks/Normal-2584.png,True,False,299,299,L,109.49,97bc28684f6ab83760f8ca3dc5f43d75
Normal-2585.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2585.png,Normal/images/Normal-2585.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2585.png,Normal/masks/Normal-2585.png,True,False,299,299,L,164.83,cf42b6771784570a097b7ccc038d1870
Normal-2586.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2586.png,Normal/images/Normal-2586.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2586.png,Normal/masks/Normal-2586.png,True,False,299,299,L,179.98,3e27ea1329d18d147f549d953bebef42
Normal-2587.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2587.png,Normal/images/Normal-2587.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2587.png,Normal/masks/Normal-2587.png,True,False,299,299,L,129.9,7da5a91f09bbe7b6db7af50ae6e26654
Normal-2588.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2588.png,Normal/images/Normal-2588.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2588.png,Normal/masks/Normal-2588.png,True,False,299,299,L,102.79,a7307f8395dc85ac5f78fa2bef2cb4d2
Normal-2589.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2589.png,Normal/images/Normal-2589.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2589.png,Normal/masks/Normal-2589.png,True,False,299,299,L,169.46,57c8a389d40d193cdfff35c53b7185d1
Normal-259.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-259.png,Normal/images/Normal-259.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-259.png,Normal/masks/Normal-259.png,True,False,299,299,L,130.79,62ced65f532a9a2e8ea23ef3428b872d
Normal-2590.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2590.png,Normal/images/Normal-2590.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2590.png,Normal/masks/Normal-2590.png,True,False,299,299,L,105.96,6b3022ae3b88dcdb8285cdedf966c207
Normal-2591.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2591.png,Normal/images/Normal-2591.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2591.png,Normal/masks/Normal-2591.png,True,False,299,299,L,135.07,d453be1eaffd3ef8cf66e69a5236032f
Normal-2592.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2592.png,Normal/images/Normal-2592.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2592.png,Normal/masks/Normal-2592.png,True,False,299,299,L,118.07,033a424cbe5babc04ce3eefbd7603dc8
Normal-2593.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2593.png,Normal/images/Normal-2593.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2593.png,Normal/masks/Normal-2593.png,True,False,299,299,L,142.27,0183ad8355837647bd71b2624bca2dc4
Normal-2594.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2594.png,Normal/images/Normal-2594.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2594.png,Normal/masks/Normal-2594.png,True,False,299,299,L,144.45,bb03beebfd497b0bb6a0d0c0ddda8a2b
Normal-2595.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2595.png,Normal/images/Normal-2595.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2595.png,Normal/masks/Normal-2595.png,True,False,299,299,L,128.53,b36f99165cdec534864291a6a5199772
Normal-2596.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2596.png,Normal/images/Normal-2596.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2596.png,Normal/masks/Normal-2596.png,True,False,299,299,L,140.97,bab213edd56c49b27593d54c4fbae563
Normal-2597.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2597.png,Normal/images/Normal-2597.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2597.png,Normal/masks/Normal-2597.png,True,False,299,299,L,154.32,72fba9eea723868acbad20bf3838c471
Normal-2598.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2598.png,Normal/images/Normal-2598.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2598.png,Normal/masks/Normal-2598.png,True,False,299,299,L,88.45,284cb080f1e63b49dc13d7181edaeed9
Normal-2599.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2599.png,Normal/images/Normal-2599.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2599.png,Normal/masks/Normal-2599.png,True,False,299,299,L,149.72,f3a326b0682e341408c3c1d7459f88c3
Normal-26.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-26.png,Normal/images/Normal-26.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-26.png,Normal/masks/Normal-26.png,True,False,299,299,L,111.21,75ef4358fd95c551e51c4c93682bd25d
Normal-260.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-260.png,Normal/images/Normal-260.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-260.png,Normal/masks/Normal-260.png,True,False,299,299,L,120.31,1a9ead4b60f26cfd9cb5a64aa3f1837b
Normal-2600.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2600.png,Normal/images/Normal-2600.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2600.png,Normal/masks/Normal-2600.png,True,False,299,299,L,116.61,72bc38c64675e190c87f7cce828ad832
Normal-2601.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2601.png,Normal/images/Normal-2601.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2601.png,Normal/masks/Normal-2601.png,True,False,299,299,L,108.0,ebde0fea9315adc1921f10b1ac8022d7
Normal-2602.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2602.png,Normal/images/Normal-2602.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2602.png,Normal/masks/Normal-2602.png,True,False,299,299,L,109.0,c9cedd145e6a7dcb46536a4497d951f4
Normal-2603.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2603.png,Normal/images/Normal-2603.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2603.png,Normal/masks/Normal-2603.png,True,False,299,299,L,155.38,61e16f57efdd05269e290e44e67ed814
Normal-2604.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2604.png,Normal/images/Normal-2604.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2604.png,Normal/masks/Normal-2604.png,True,False,299,299,L,116.67,ac69713fa297123fb534c73e6eef6e7c
Normal-2605.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2605.png,Normal/images/Normal-2605.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2605.png,Normal/masks/Normal-2605.png,True,False,299,299,L,52.04,df6a2dec3775c56e9dc6731bcc3c44fa
Normal-2606.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2606.png,Normal/images/Normal-2606.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2606.png,Normal/masks/Normal-2606.png,True,False,299,299,L,94.07,4b2569bc0b5784e4d1a13585049d96a9
Normal-2607.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2607.png,Normal/images/Normal-2607.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2607.png,Normal/masks/Normal-2607.png,True,False,299,299,L,130.15,4ca2e440e41259e16bd40c54912a2296
Normal-2608.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2608.png,Normal/images/Normal-2608.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2608.png,Normal/masks/Normal-2608.png,True,False,299,299,L,125.91,1b0c27882e59459abae74277e9a61c38
Normal-2609.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2609.png,Normal/images/Normal-2609.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2609.png,Normal/masks/Normal-2609.png,True,False,299,299,L,150.35,eef19f40ff56c87b74d4bc2711e133c8
Normal-261.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-261.png,Normal/images/Normal-261.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-261.png,Normal/masks/Normal-261.png,True,False,299,299,L,101.98,ec8c5d1ca9392699de0e178c26978142
Normal-2610.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2610.png,Normal/images/Normal-2610.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2610.png,Normal/masks/Normal-2610.png,True,False,299,299,L,118.21,e007d5607f18848eb4c170c400f77880
Normal-2611.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2611.png,Normal/images/Normal-2611.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2611.png,Normal/masks/Normal-2611.png,True,False,299,299,L,127.79,9aad8543431170142aca71f56a56edde
Normal-2612.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2612.png,Normal/images/Normal-2612.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2612.png,Normal/masks/Normal-2612.png,True,False,299,299,L,124.83,28080a98b2f673bf3f43df4b6dfb53db
Normal-2613.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2613.png,Normal/images/Normal-2613.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2613.png,Normal/masks/Normal-2613.png,True,False,299,299,L,90.54,7f8fe471cdee0eaed9e07f014438f042
Normal-2614.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2614.png,Normal/images/Normal-2614.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2614.png,Normal/masks/Normal-2614.png,True,False,299,299,L,113.72,a093c0b752832d3020cd7dde3b4507ca
Normal-2615.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2615.png,Normal/images/Normal-2615.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2615.png,Normal/masks/Normal-2615.png,True,False,299,299,L,125.98,d780fb0d0c1d5e1b8b5b4960e0e85022
Normal-2616.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2616.png,Normal/images/Normal-2616.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2616.png,Normal/masks/Normal-2616.png,True,False,299,299,L,118.61,2cd2aa2587f09bc677e0149cc64334cd
Normal-2617.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2617.png,Normal/images/Normal-2617.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2617.png,Normal/masks/Normal-2617.png,True,False,299,299,L,105.41,0cacdd3374717c347d197bbcc8f808a9
Normal-2618.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2618.png,Normal/images/Normal-2618.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2618.png,Normal/masks/Normal-2618.png,True,False,299,299,L,158.25,e8d5908d0c3aac862921d2221ee46995
Normal-2619.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2619.png,Normal/images/Normal-2619.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2619.png,Normal/masks/Normal-2619.png,True,False,299,299,L,146.97,c1e9b650eef3d87ebce337c9ee886ef3
Normal-262.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-262.png,Normal/images/Normal-262.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-262.png,Normal/masks/Normal-262.png,True,False,299,299,L,91.66,468233991db2fb95fe20a346bb256c40
Normal-2620.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2620.png,Normal/images/Normal-2620.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2620.png,Normal/masks/Normal-2620.png,True,False,299,299,L,131.75,e0f3ae833f45581bd7ff9db2f783aac0
Normal-2621.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2621.png,Normal/images/Normal-2621.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2621.png,Normal/masks/Normal-2621.png,True,False,299,299,L,115.69,f625bc5a81811d676b1019c57c00045f
Normal-2622.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2622.png,Normal/images/Normal-2622.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2622.png,Normal/masks/Normal-2622.png,True,False,299,299,L,127.93,fb9cdea3f1bd963fc9f719ad25cc4ad7
Normal-2623.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2623.png,Normal/images/Normal-2623.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2623.png,Normal/masks/Normal-2623.png,True,False,299,299,L,114.85,0bac87c57f32f1df5a9300715a8cea54
Normal-2624.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2624.png,Normal/images/Normal-2624.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2624.png,Normal/masks/Normal-2624.png,True,False,299,299,L,158.04,326c269edd2aaf19d52fa50fc378cab7
Normal-2625.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2625.png,Normal/images/Normal-2625.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2625.png,Normal/masks/Normal-2625.png,True,False,299,299,L,153.25,1b0a01e0f19789745e96aa748500ef67
Normal-2626.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2626.png,Normal/images/Normal-2626.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2626.png,Normal/masks/Normal-2626.png,True,False,299,299,L,157.28,6d44696b3fb1b7fe15e180060d5dbf20
Normal-2627.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2627.png,Normal/images/Normal-2627.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2627.png,Normal/masks/Normal-2627.png,True,False,299,299,L,177.65,01de51f0e65af31edb03151a00fa28c9
Normal-2628.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2628.png,Normal/images/Normal-2628.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2628.png,Normal/masks/Normal-2628.png,True,False,299,299,L,99.51,7f629afefa5a1d7a80f0663fbd485b81
Normal-2629.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2629.png,Normal/images/Normal-2629.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2629.png,Normal/masks/Normal-2629.png,True,False,299,299,L,117.94,320ccdfd52b44f99d16d1027ad88ab62
Normal-263.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-263.png,Normal/images/Normal-263.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-263.png,Normal/masks/Normal-263.png,True,False,299,299,L,113.52,8dd3a4da466ee99b52c97e5f5e486933
Normal-2630.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2630.png,Normal/images/Normal-2630.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2630.png,Normal/masks/Normal-2630.png,True,False,299,299,L,149.72,8724444bb85d07c26fd22e0d65adb3c5
Normal-2631.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2631.png,Normal/images/Normal-2631.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2631.png,Normal/masks/Normal-2631.png,True,False,299,299,L,171.16,9bb135673f1419a728cec0145b0663d0
Normal-2632.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2632.png,Normal/images/Normal-2632.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2632.png,Normal/masks/Normal-2632.png,True,False,299,299,L,126.44,d3cb9f9758d76410c976b66c9da8618e
Normal-2633.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2633.png,Normal/images/Normal-2633.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2633.png,Normal/masks/Normal-2633.png,True,False,299,299,L,174.93,56914e202a6986f6db3414cb9a4735b6
Normal-2634.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2634.png,Normal/images/Normal-2634.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2634.png,Normal/masks/Normal-2634.png,True,False,299,299,L,136.2,6b15bb39f547089a506104f9819c46e2
Normal-2635.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2635.png,Normal/images/Normal-2635.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2635.png,Normal/masks/Normal-2635.png,True,False,299,299,L,102.46,22d9977e3502d085aedcecf3fa01d3a5
Normal-2636.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2636.png,Normal/images/Normal-2636.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2636.png,Normal/masks/Normal-2636.png,True,False,299,299,L,159.79,3cf03c3d7af4167be51a4544717f91ae
Normal-2637.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2637.png,Normal/images/Normal-2637.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2637.png,Normal/masks/Normal-2637.png,True,False,299,299,L,119.95,834fd0e29719577db6efd4505a157a1b
Normal-2638.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2638.png,Normal/images/Normal-2638.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2638.png,Normal/masks/Normal-2638.png,True,False,299,299,L,110.43,a90aa0da961c6809e39a633f342fcf0e
Normal-2639.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2639.png,Normal/images/Normal-2639.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2639.png,Normal/masks/Normal-2639.png,True,False,299,299,L,186.98,257f8a12835e8d80ad74d698448471df
Normal-264.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-264.png,Normal/images/Normal-264.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-264.png,Normal/masks/Normal-264.png,True,False,299,299,L,125.44,aa852e8bb2b816c307b5ad39e38f11f9
Normal-2640.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2640.png,Normal/images/Normal-2640.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2640.png,Normal/masks/Normal-2640.png,True,False,299,299,L,160.59,45e2f8e7c5c313c99a1445925cb67c93
Normal-2641.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2641.png,Normal/images/Normal-2641.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2641.png,Normal/masks/Normal-2641.png,True,False,299,299,L,121.21,58ff87c638726423f5decb764895846a
Normal-2642.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2642.png,Normal/images/Normal-2642.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2642.png,Normal/masks/Normal-2642.png,True,False,299,299,L,106.13,9197c96bb9f694dfe112ffe84e4fd927
Normal-2643.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2643.png,Normal/images/Normal-2643.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2643.png,Normal/masks/Normal-2643.png,True,False,299,299,L,111.57,504c9da31dfc7539211ab15a9c7b776f
Normal-2644.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2644.png,Normal/images/Normal-2644.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2644.png,Normal/masks/Normal-2644.png,True,False,299,299,L,93.96,b6677a79395c0f4783053f71cec20b05
Normal-2645.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2645.png,Normal/images/Normal-2645.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2645.png,Normal/masks/Normal-2645.png,True,False,299,299,L,151.92,9a5863f3f536bee253ba27eb7be1b4da
Normal-2646.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2646.png,Normal/images/Normal-2646.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2646.png,Normal/masks/Normal-2646.png,True,False,299,299,L,139.17,e30088c04cb8cd116d411ce5df8999b4
Normal-2647.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2647.png,Normal/images/Normal-2647.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2647.png,Normal/masks/Normal-2647.png,True,False,299,299,L,62.59,f031fc7245e45a2e1a6d6f47887db554
Normal-2648.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2648.png,Normal/images/Normal-2648.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2648.png,Normal/masks/Normal-2648.png,True,False,299,299,L,110.56,12f393c9f2b2b800a266b0bff67f0403
Normal-2649.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2649.png,Normal/images/Normal-2649.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2649.png,Normal/masks/Normal-2649.png,True,False,299,299,L,100.31,6f3e1aa48f7cc7055a4c259c146c87cb
Normal-265.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-265.png,Normal/images/Normal-265.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-265.png,Normal/masks/Normal-265.png,True,False,299,299,L,124.37,4bdc14c1d7632ef1376cd2e31862ac3c
Normal-2650.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2650.png,Normal/images/Normal-2650.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2650.png,Normal/masks/Normal-2650.png,True,False,299,299,L,165.86,36391b96a01e3bb31d7aec6ce9b83ca6
Normal-2651.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2651.png,Normal/images/Normal-2651.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2651.png,Normal/masks/Normal-2651.png,True,False,299,299,L,160.73,e76a6c8e484fd639aada4532300f065c
Normal-2652.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2652.png,Normal/images/Normal-2652.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2652.png,Normal/masks/Normal-2652.png,True,False,299,299,L,99.05,716936a501f712a802d8991698595a51
Normal-2653.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2653.png,Normal/images/Normal-2653.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2653.png,Normal/masks/Normal-2653.png,True,False,299,299,L,142.63,0deb9eb19becafe80ee81f9e4a1f38aa
Normal-2654.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2654.png,Normal/images/Normal-2654.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2654.png,Normal/masks/Normal-2654.png,True,False,299,299,L,119.05,ecedd8faa0b54f5c48f0a21bbf75a6be
Normal-2655.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2655.png,Normal/images/Normal-2655.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2655.png,Normal/masks/Normal-2655.png,True,False,299,299,L,112.24,89c03b2eede7a871ad8dc8825a9d5b1c
Normal-2656.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2656.png,Normal/images/Normal-2656.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2656.png,Normal/masks/Normal-2656.png,True,False,299,299,L,89.8,e719bf38ed6dd8eef6582c8c7d6bdc1e
Normal-2657.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2657.png,Normal/images/Normal-2657.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2657.png,Normal/masks/Normal-2657.png,True,False,299,299,L,148.3,11e6fcc66f2e2c645a8b89689f31f5ca
Normal-2658.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2658.png,Normal/images/Normal-2658.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2658.png,Normal/masks/Normal-2658.png,True,False,299,299,L,171.89,9cf58018f7b295101f31cb9af7153cbb
Normal-2659.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2659.png,Normal/images/Normal-2659.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2659.png,Normal/masks/Normal-2659.png,True,False,299,299,L,113.47,c45769f99d9aa50f782496ab4312ce3d
Normal-266.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-266.png,Normal/images/Normal-266.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-266.png,Normal/masks/Normal-266.png,True,False,299,299,L,114.58,1d592ea1222d96fc68559bac06c70c6c
Normal-2660.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2660.png,Normal/images/Normal-2660.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2660.png,Normal/masks/Normal-2660.png,True,False,299,299,L,112.58,dac91ab7f42c61b4a174c08042b0487e
Normal-2661.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2661.png,Normal/images/Normal-2661.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2661.png,Normal/masks/Normal-2661.png,True,False,299,299,L,106.89,418bd178bdbf283166d8e9064f1bf8b7
Normal-2662.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2662.png,Normal/images/Normal-2662.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2662.png,Normal/masks/Normal-2662.png,True,False,299,299,L,163.91,6334449313e8c2bc7f028df88271255b
Normal-2663.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2663.png,Normal/images/Normal-2663.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2663.png,Normal/masks/Normal-2663.png,True,False,299,299,L,95.03,b24d3b7628c1abc3c1df266997aa51f5
Normal-2664.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2664.png,Normal/images/Normal-2664.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2664.png,Normal/masks/Normal-2664.png,True,False,299,299,L,99.71,e3a2397cfa8fd75c9165e7924f02702e
Normal-2665.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2665.png,Normal/images/Normal-2665.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2665.png,Normal/masks/Normal-2665.png,True,False,299,299,L,124.44,84dc05fd0925a2792df6e57bd1cdd2db
Normal-2666.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2666.png,Normal/images/Normal-2666.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2666.png,Normal/masks/Normal-2666.png,True,False,299,299,L,159.57,2dca005936b6b3cfb91f877df6348a2a
Normal-2667.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2667.png,Normal/images/Normal-2667.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2667.png,Normal/masks/Normal-2667.png,True,False,299,299,L,142.4,02efa76be0c30d122546c25eac937554
Normal-2668.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2668.png,Normal/images/Normal-2668.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2668.png,Normal/masks/Normal-2668.png,True,False,299,299,L,121.99,6a7c50ab90408587e3e29d1be565b7a0
Normal-2669.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2669.png,Normal/images/Normal-2669.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2669.png,Normal/masks/Normal-2669.png,True,False,299,299,L,134.45,535b6c6c1a3c337b288ca8768a4b3f59
Normal-267.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-267.png,Normal/images/Normal-267.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-267.png,Normal/masks/Normal-267.png,True,False,299,299,L,135.84,db579cce282f682bf5c3acff5180e60d
Normal-2670.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2670.png,Normal/images/Normal-2670.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2670.png,Normal/masks/Normal-2670.png,True,False,299,299,L,133.17,b4dfb366c45f1b674927e386f2145d3d
Normal-2671.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2671.png,Normal/images/Normal-2671.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2671.png,Normal/masks/Normal-2671.png,True,False,299,299,L,143.58,4d70d8c10ccc2664b0843742c1db1757
Normal-2672.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2672.png,Normal/images/Normal-2672.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2672.png,Normal/masks/Normal-2672.png,True,False,299,299,L,110.83,0f6f3ffa06816702987d4609066f4603
Normal-2673.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2673.png,Normal/images/Normal-2673.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2673.png,Normal/masks/Normal-2673.png,True,False,299,299,L,168.49,e6e71b1bc130ecafbcc73db87d9171c7
Normal-2674.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2674.png,Normal/images/Normal-2674.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2674.png,Normal/masks/Normal-2674.png,True,False,299,299,L,113.72,a52cc9325b71923f66956be6d7d9f38d
Normal-2675.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2675.png,Normal/images/Normal-2675.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2675.png,Normal/masks/Normal-2675.png,True,False,299,299,L,115.55,502dc5c119576ed8d1a1967e8592b76e
Normal-2676.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2676.png,Normal/images/Normal-2676.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2676.png,Normal/masks/Normal-2676.png,True,False,299,299,L,157.79,2e81afa488d2be6418963629d37f94f5
Normal-2677.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2677.png,Normal/images/Normal-2677.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2677.png,Normal/masks/Normal-2677.png,True,False,299,299,L,145.96,692a1323f5ee436a9d25361246c57d3b
Normal-2678.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2678.png,Normal/images/Normal-2678.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2678.png,Normal/masks/Normal-2678.png,True,False,299,299,L,121.5,f7ee4b9b6fcfc7a2be578f6a9c39a593
Normal-2679.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2679.png,Normal/images/Normal-2679.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2679.png,Normal/masks/Normal-2679.png,True,False,299,299,L,141.22,478bbf940bb0bca561f56f05c55c7906
Normal-268.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-268.png,Normal/images/Normal-268.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-268.png,Normal/masks/Normal-268.png,True,False,299,299,L,135.99,5a2f09c06cb7ab448e660483adae8570
Normal-2680.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2680.png,Normal/images/Normal-2680.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2680.png,Normal/masks/Normal-2680.png,True,False,299,299,L,119.28,61858ee1f52a04a3cdc68d676132dcef
Normal-2681.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2681.png,Normal/images/Normal-2681.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2681.png,Normal/masks/Normal-2681.png,True,False,299,299,L,117.33,9c672ec163ebc0c81d5fd69610abbd70
Normal-2682.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2682.png,Normal/images/Normal-2682.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2682.png,Normal/masks/Normal-2682.png,True,False,299,299,L,164.34,6dd61457d07699a3529b8cea97e0f362
Normal-2683.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2683.png,Normal/images/Normal-2683.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2683.png,Normal/masks/Normal-2683.png,True,False,299,299,L,115.45,8d9aec44f9977172c829e55babd2cc10
Normal-2684.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2684.png,Normal/images/Normal-2684.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2684.png,Normal/masks/Normal-2684.png,True,False,299,299,L,120.16,b98223c6cfafeb684cdad95842b7109f
Normal-2685.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2685.png,Normal/images/Normal-2685.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2685.png,Normal/masks/Normal-2685.png,True,False,299,299,L,99.62,361d9710209541a7690f3eeeb9f6088d
Normal-2686.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2686.png,Normal/images/Normal-2686.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2686.png,Normal/masks/Normal-2686.png,True,False,299,299,L,145.89,f416e21d8ef7720382af016edf2fac0a
Normal-2687.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2687.png,Normal/images/Normal-2687.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2687.png,Normal/masks/Normal-2687.png,True,False,299,299,L,178.02,aee6f917a724f9c8bef36dda7d0d0c9b
Normal-2688.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2688.png,Normal/images/Normal-2688.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2688.png,Normal/masks/Normal-2688.png,True,False,299,299,L,146.08,54384845d9682c89aa96c1df037a19d4
Normal-2689.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2689.png,Normal/images/Normal-2689.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2689.png,Normal/masks/Normal-2689.png,True,False,299,299,L,121.18,4f11c010e7ef6af0a32f06bf0a3f6a81
Normal-269.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-269.png,Normal/images/Normal-269.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-269.png,Normal/masks/Normal-269.png,True,False,299,299,L,121.54,a68a55431fc561e3c8faaa370a7c4c98
Normal-2690.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2690.png,Normal/images/Normal-2690.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2690.png,Normal/masks/Normal-2690.png,True,False,299,299,L,133.74,60f9413d963c3390ddf0e10e9b52d5f4
Normal-2691.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2691.png,Normal/images/Normal-2691.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2691.png,Normal/masks/Normal-2691.png,True,False,299,299,L,135.23,f3127ef32f150eb2d90c43ea33d0173f
Normal-2692.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2692.png,Normal/images/Normal-2692.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2692.png,Normal/masks/Normal-2692.png,True,False,299,299,L,123.48,2e8119907cc5dfce8e566d9b5a666855
Normal-2693.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2693.png,Normal/images/Normal-2693.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2693.png,Normal/masks/Normal-2693.png,True,False,299,299,L,138.1,7e67f7ae5b9cc207a1c243035560dc3b
Normal-2694.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2694.png,Normal/images/Normal-2694.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2694.png,Normal/masks/Normal-2694.png,True,False,299,299,L,111.94,a052b44b31979f8a447a78c5b9b955f5
Normal-2695.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2695.png,Normal/images/Normal-2695.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2695.png,Normal/masks/Normal-2695.png,True,False,299,299,L,119.65,e81bd6d1ac6e85d85fe7b4cdfc320c6e
Normal-2696.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2696.png,Normal/images/Normal-2696.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2696.png,Normal/masks/Normal-2696.png,True,False,299,299,L,163.42,2779109c139d17c61d453b6df40b2ae4
Normal-2697.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2697.png,Normal/images/Normal-2697.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2697.png,Normal/masks/Normal-2697.png,True,False,299,299,L,118.14,73de0363f7549a4543128d3884247023
Normal-2698.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2698.png,Normal/images/Normal-2698.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2698.png,Normal/masks/Normal-2698.png,True,False,299,299,L,133.34,8e8556537465235e8a9b7a37bfbb05ba
Normal-2699.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2699.png,Normal/images/Normal-2699.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2699.png,Normal/masks/Normal-2699.png,True,False,299,299,L,131.74,d0f1103505549aba58efb71df58dc8d4
Normal-27.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-27.png,Normal/images/Normal-27.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-27.png,Normal/masks/Normal-27.png,True,False,299,299,L,162.07,1346d143e212564c083edf7e16aa2c4e
Normal-270.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-270.png,Normal/images/Normal-270.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-270.png,Normal/masks/Normal-270.png,True,False,299,299,L,123.37,4059bc1ec4ecf240656ca8aa83b92145
Normal-2700.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2700.png,Normal/images/Normal-2700.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2700.png,Normal/masks/Normal-2700.png,True,False,299,299,L,126.35,32975fc8fec699effe7285ea042bfc4d
Normal-2701.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2701.png,Normal/images/Normal-2701.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2701.png,Normal/masks/Normal-2701.png,True,False,299,299,L,103.05,91d0e4bfdb015dcbae83bb40c5ad10dc
Normal-2702.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2702.png,Normal/images/Normal-2702.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2702.png,Normal/masks/Normal-2702.png,True,False,299,299,L,127.3,41fd8cbf3b609be6668761c0d20fa0ed
Normal-2703.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2703.png,Normal/images/Normal-2703.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2703.png,Normal/masks/Normal-2703.png,True,False,299,299,L,114.63,44fe3d7662ad2fe1a2fc7c142d412c96
Normal-2704.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2704.png,Normal/images/Normal-2704.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2704.png,Normal/masks/Normal-2704.png,True,False,299,299,L,173.45,c53ae10d39548d899eb8a1f1f7cc489a
Normal-2705.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2705.png,Normal/images/Normal-2705.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2705.png,Normal/masks/Normal-2705.png,True,False,299,299,L,109.35,e350bc67e3463c04f4b89b3d527b6567
Normal-2706.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2706.png,Normal/images/Normal-2706.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2706.png,Normal/masks/Normal-2706.png,True,False,299,299,L,158.82,e9181fee552e24a01895367a32f6886d
Normal-2707.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2707.png,Normal/images/Normal-2707.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2707.png,Normal/masks/Normal-2707.png,True,False,299,299,L,128.0,aa8ccd7c8e6f53092bebd706ea80b0ef
Normal-2708.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2708.png,Normal/images/Normal-2708.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2708.png,Normal/masks/Normal-2708.png,True,False,299,299,L,125.05,273ff04c02821fa951c1429296bf14e8
Normal-2709.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2709.png,Normal/images/Normal-2709.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2709.png,Normal/masks/Normal-2709.png,True,False,299,299,L,160.4,af0f71d61a31e0b645ef680b3456c4fc
Normal-271.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-271.png,Normal/images/Normal-271.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-271.png,Normal/masks/Normal-271.png,True,False,299,299,L,131.6,d68ea33d59a63a4653f3ac4285dc5a4e
Normal-2710.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2710.png,Normal/images/Normal-2710.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2710.png,Normal/masks/Normal-2710.png,True,False,299,299,L,128.2,f813671d432d3c819a23d14b249b6e89
Normal-2711.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2711.png,Normal/images/Normal-2711.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2711.png,Normal/masks/Normal-2711.png,True,False,299,299,L,134.86,4c85d38cbd2a4e8da94acd6ddf207090
Normal-2712.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2712.png,Normal/images/Normal-2712.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2712.png,Normal/masks/Normal-2712.png,True,False,299,299,L,126.09,cde41a0e2cc6ea5002430f8f648b3e38
Normal-2713.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2713.png,Normal/images/Normal-2713.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2713.png,Normal/masks/Normal-2713.png,True,False,299,299,L,170.61,56b794d8082c71a7c7332cd443d5d920
Normal-2714.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2714.png,Normal/images/Normal-2714.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2714.png,Normal/masks/Normal-2714.png,True,False,299,299,L,128.01,8d77199da3ccacf118b78fde99097a37
Normal-2715.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2715.png,Normal/images/Normal-2715.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2715.png,Normal/masks/Normal-2715.png,True,False,299,299,L,103.61,f20eaf8aaaa37f8dc8ea0f0cd5b60f6a
Normal-2716.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2716.png,Normal/images/Normal-2716.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2716.png,Normal/masks/Normal-2716.png,True,False,299,299,L,164.7,0437f7f755b7db469a4fc8c17df01276
Normal-2717.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2717.png,Normal/images/Normal-2717.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2717.png,Normal/masks/Normal-2717.png,True,False,299,299,L,132.88,4416541c0f6da6264c8b062c79f2f4b8
Normal-2718.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2718.png,Normal/images/Normal-2718.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2718.png,Normal/masks/Normal-2718.png,True,False,299,299,L,124.63,da2b2a2d8bfeff7539f39fd17bfb2ae1
Normal-2719.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2719.png,Normal/images/Normal-2719.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2719.png,Normal/masks/Normal-2719.png,True,False,299,299,L,162.39,c3035b422ba765b1330e754b32d54382
Normal-272.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-272.png,Normal/images/Normal-272.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-272.png,Normal/masks/Normal-272.png,True,False,299,299,L,144.57,b4a5415c0f09fa5911cdae96b36a2151
Normal-2720.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2720.png,Normal/images/Normal-2720.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2720.png,Normal/masks/Normal-2720.png,True,False,299,299,L,118.44,5a938636fb4c00cef14ec73559d778a7
Normal-2721.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2721.png,Normal/images/Normal-2721.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2721.png,Normal/masks/Normal-2721.png,True,False,299,299,L,110.05,54e716021e58dca3cb3a18864c22e6a2
Normal-2722.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2722.png,Normal/images/Normal-2722.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2722.png,Normal/masks/Normal-2722.png,True,False,299,299,L,124.43,226e553916301b6bc7c0baae1cd2df46
Normal-2723.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2723.png,Normal/images/Normal-2723.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2723.png,Normal/masks/Normal-2723.png,True,False,299,299,L,117.58,a43503a155cf5a98360d43b835704dcd
Normal-2724.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2724.png,Normal/images/Normal-2724.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2724.png,Normal/masks/Normal-2724.png,True,False,299,299,L,137.15,689280db56f64dfd840d149ae5b7e9f4
Normal-2725.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2725.png,Normal/images/Normal-2725.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2725.png,Normal/masks/Normal-2725.png,True,False,299,299,L,120.14,5b55856d5fad00f1965e1d724c1b3070
Normal-2726.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2726.png,Normal/images/Normal-2726.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2726.png,Normal/masks/Normal-2726.png,True,False,299,299,L,108.75,43cabf1a208566bc3b612d354ce97f2a
Normal-2727.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2727.png,Normal/images/Normal-2727.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2727.png,Normal/masks/Normal-2727.png,True,False,299,299,L,134.09,85f53ab59cc6cd562bc9c341a5cf6975
Normal-2728.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2728.png,Normal/images/Normal-2728.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2728.png,Normal/masks/Normal-2728.png,True,False,299,299,L,105.26,fcb987ac916a97b373a5bdeed00a72ef
Normal-2729.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2729.png,Normal/images/Normal-2729.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2729.png,Normal/masks/Normal-2729.png,True,False,299,299,L,119.33,d22df3ff1ef05a9b2388953a9dcdc744
Normal-273.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-273.png,Normal/images/Normal-273.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-273.png,Normal/masks/Normal-273.png,True,False,299,299,L,126.1,d2cac81274aa0b745683a03ce922c77e
Normal-2730.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2730.png,Normal/images/Normal-2730.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2730.png,Normal/masks/Normal-2730.png,True,False,299,299,L,125.58,16d5c5431d57449c532c6469254a0cee
Normal-2731.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2731.png,Normal/images/Normal-2731.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2731.png,Normal/masks/Normal-2731.png,True,False,299,299,L,141.74,869e025c0f9d5ea3d3d20a43e075a4b8
Normal-2732.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2732.png,Normal/images/Normal-2732.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2732.png,Normal/masks/Normal-2732.png,True,False,299,299,L,115.54,3444da7902ef866cd3670553cc193b26
Normal-2733.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2733.png,Normal/images/Normal-2733.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2733.png,Normal/masks/Normal-2733.png,True,False,299,299,L,180.04,c6dcc28507fd52d1ec5804efc47e1657
Normal-2734.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2734.png,Normal/images/Normal-2734.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2734.png,Normal/masks/Normal-2734.png,True,False,299,299,L,122.43,ea55d81af506fa0ddfa1be6e7656205f
Normal-2735.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2735.png,Normal/images/Normal-2735.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2735.png,Normal/masks/Normal-2735.png,True,False,299,299,L,172.7,930721167bc80f23d1b9fed507b1215d
Normal-2736.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2736.png,Normal/images/Normal-2736.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2736.png,Normal/masks/Normal-2736.png,True,False,299,299,L,165.89,b2a590352112b1f65b137e9ad07c5f33
Normal-2737.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2737.png,Normal/images/Normal-2737.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2737.png,Normal/masks/Normal-2737.png,True,False,299,299,L,124.7,3e968092fcba15a42b90eb4544dbd79b
Normal-2738.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2738.png,Normal/images/Normal-2738.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2738.png,Normal/masks/Normal-2738.png,True,False,299,299,L,139.77,d668fd4a74b005559d752d949327fb77
Normal-2739.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2739.png,Normal/images/Normal-2739.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2739.png,Normal/masks/Normal-2739.png,True,False,299,299,L,99.76,19c422f6f1a32fc09d62f3010383a25c
Normal-274.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-274.png,Normal/images/Normal-274.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-274.png,Normal/masks/Normal-274.png,True,False,299,299,L,113.08,71a18ff9116db9a0324c510834291694
Normal-2740.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2740.png,Normal/images/Normal-2740.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2740.png,Normal/masks/Normal-2740.png,True,False,299,299,L,127.84,f7d865ade18dbccd34694a754308240b
Normal-2741.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2741.png,Normal/images/Normal-2741.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2741.png,Normal/masks/Normal-2741.png,True,False,299,299,L,112.97,8756fdf92aba51758ed714f21cf5adb6
Normal-2742.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2742.png,Normal/images/Normal-2742.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2742.png,Normal/masks/Normal-2742.png,True,False,299,299,L,167.7,443b4e555652d06fe81278da9baeef56
Normal-2743.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2743.png,Normal/images/Normal-2743.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2743.png,Normal/masks/Normal-2743.png,True,False,299,299,L,104.15,b840f1ea1758a24b20f44f92ba3b3fd1
Normal-2744.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2744.png,Normal/images/Normal-2744.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2744.png,Normal/masks/Normal-2744.png,True,False,299,299,L,123.47,c1145720056dd4567224a5838650e91e
Normal-2745.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2745.png,Normal/images/Normal-2745.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2745.png,Normal/masks/Normal-2745.png,True,False,299,299,L,156.01,d5e91ccb159cb35e097ddf0b699fa45c
Normal-2746.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2746.png,Normal/images/Normal-2746.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2746.png,Normal/masks/Normal-2746.png,True,False,299,299,L,159.93,a6d89580ff6ca21677db88d161253e4f
Normal-2747.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2747.png,Normal/images/Normal-2747.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2747.png,Normal/masks/Normal-2747.png,True,False,299,299,L,143.42,89b1cf65c7f68a0804458dfad69d9da2
Normal-2748.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2748.png,Normal/images/Normal-2748.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2748.png,Normal/masks/Normal-2748.png,True,False,299,299,L,94.04,fd9b8e65a943e59c4a0808e8f67abb81
Normal-2749.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2749.png,Normal/images/Normal-2749.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2749.png,Normal/masks/Normal-2749.png,True,False,299,299,L,171.17,e2299e2ef11e21fd05f17c04ef7dddf5
Normal-275.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-275.png,Normal/images/Normal-275.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-275.png,Normal/masks/Normal-275.png,True,False,299,299,L,115.16,f2562b4280c2f6adea521c4991b0782f
Normal-2750.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2750.png,Normal/images/Normal-2750.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2750.png,Normal/masks/Normal-2750.png,True,False,299,299,L,135.99,2f6229b20b71fcca0923d0c80cf953e3
Normal-2751.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2751.png,Normal/images/Normal-2751.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2751.png,Normal/masks/Normal-2751.png,True,False,299,299,L,160.34,67603be9c6dabd6bf4ee36a2fa93abf3
Normal-2752.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2752.png,Normal/images/Normal-2752.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2752.png,Normal/masks/Normal-2752.png,True,False,299,299,L,167.03,5dc141102806c464bcd1adccf5ec1a03
Normal-2753.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2753.png,Normal/images/Normal-2753.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2753.png,Normal/masks/Normal-2753.png,True,False,299,299,L,131.28,e4f7a2b2a9049eeddf90783890238c4b
Normal-2754.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2754.png,Normal/images/Normal-2754.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2754.png,Normal/masks/Normal-2754.png,True,False,299,299,L,107.95,0c8269a2e592c31f7d6b46a2107cb0fe
Normal-2755.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2755.png,Normal/images/Normal-2755.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2755.png,Normal/masks/Normal-2755.png,True,False,299,299,L,145.8,026a94acb3f27423b720adaeeebc3f28
Normal-2756.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2756.png,Normal/images/Normal-2756.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2756.png,Normal/masks/Normal-2756.png,True,False,299,299,L,183.2,49d321022cd5fbf0e5f70b3df27430ea
Normal-2757.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2757.png,Normal/images/Normal-2757.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2757.png,Normal/masks/Normal-2757.png,True,False,299,299,L,118.17,3aafff79e3d90a2f94398effafefabbe
Normal-2758.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2758.png,Normal/images/Normal-2758.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2758.png,Normal/masks/Normal-2758.png,True,False,299,299,L,110.53,641453c49a2992513577179f49fb4d00
Normal-2759.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2759.png,Normal/images/Normal-2759.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2759.png,Normal/masks/Normal-2759.png,True,False,299,299,L,132.45,ba9476d15876ae8fe9320d4b7928ad03
Normal-276.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-276.png,Normal/images/Normal-276.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-276.png,Normal/masks/Normal-276.png,True,False,299,299,L,118.67,bcbe74b049fa656e5ebb27a39efbebb0
Normal-2760.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2760.png,Normal/images/Normal-2760.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2760.png,Normal/masks/Normal-2760.png,True,False,299,299,L,129.89,926d1ab90e0f9bfcd5e58c15860d9def
Normal-2761.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2761.png,Normal/images/Normal-2761.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2761.png,Normal/masks/Normal-2761.png,True,False,299,299,L,170.76,def6a7767f82bf56c8ca729e65c0544e
Normal-2762.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2762.png,Normal/images/Normal-2762.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2762.png,Normal/masks/Normal-2762.png,True,False,299,299,L,107.11,918155eb891465101567a02cf481247d
Normal-2763.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2763.png,Normal/images/Normal-2763.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2763.png,Normal/masks/Normal-2763.png,True,False,299,299,L,115.83,38f976d3a6f4093cbf14ca012c26f42e
Normal-2764.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2764.png,Normal/images/Normal-2764.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2764.png,Normal/masks/Normal-2764.png,True,False,299,299,L,117.78,a0cf97074ee0b633ed1ce3050a19d48e
Normal-2765.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2765.png,Normal/images/Normal-2765.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2765.png,Normal/masks/Normal-2765.png,True,False,299,299,L,166.51,ba3056048a7d96af5924d45a9da06f60
Normal-2766.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2766.png,Normal/images/Normal-2766.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2766.png,Normal/masks/Normal-2766.png,True,False,299,299,L,124.34,5f17d3599f39fe3bc1bff2ed4b9a812e
Normal-2767.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2767.png,Normal/images/Normal-2767.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2767.png,Normal/masks/Normal-2767.png,True,False,299,299,L,145.57,936c0cd356a76add24cf0a3a60e73650
Normal-2768.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2768.png,Normal/images/Normal-2768.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2768.png,Normal/masks/Normal-2768.png,True,False,299,299,L,114.27,d36f1c31729cd226fdc412e303a1fcc6
Normal-2769.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2769.png,Normal/images/Normal-2769.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2769.png,Normal/masks/Normal-2769.png,True,False,299,299,L,175.44,49d30bca01434323e4097427aedb7c22
Normal-277.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-277.png,Normal/images/Normal-277.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-277.png,Normal/masks/Normal-277.png,True,False,299,299,L,128.35,e138c201b371b4d24e30a0171c649e08
Normal-2770.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2770.png,Normal/images/Normal-2770.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2770.png,Normal/masks/Normal-2770.png,True,False,299,299,L,109.93,4a114583a1c7fdb7ec3d02cbc51d9cd5
Normal-2771.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2771.png,Normal/images/Normal-2771.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2771.png,Normal/masks/Normal-2771.png,True,False,299,299,L,134.47,0ba7ac1927a6cdaf0ef6d0580091cdbb
Normal-2772.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2772.png,Normal/images/Normal-2772.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2772.png,Normal/masks/Normal-2772.png,True,False,299,299,L,140.21,00db57281b54f752c8e4292bd3007776
Normal-2773.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2773.png,Normal/images/Normal-2773.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2773.png,Normal/masks/Normal-2773.png,True,False,299,299,L,108.63,c74aad77fabc74af850b99d3c8e63b27
Normal-2774.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2774.png,Normal/images/Normal-2774.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2774.png,Normal/masks/Normal-2774.png,True,False,299,299,L,114.55,ca69c447d998d85a504d909fe3a7f8fa
Normal-2775.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2775.png,Normal/images/Normal-2775.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2775.png,Normal/masks/Normal-2775.png,True,False,299,299,L,122.65,fdf17164ba9c9f86b4565a84dbc0bbde
Normal-2776.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2776.png,Normal/images/Normal-2776.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2776.png,Normal/masks/Normal-2776.png,True,False,299,299,L,113.53,a28c2f6add0c6e6e01178d73cb78f640
Normal-2777.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2777.png,Normal/images/Normal-2777.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2777.png,Normal/masks/Normal-2777.png,True,False,299,299,L,175.69,396e5aac7d7d3b737c969e3b331544fb
Normal-2778.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2778.png,Normal/images/Normal-2778.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2778.png,Normal/masks/Normal-2778.png,True,False,299,299,L,116.81,e83982e065e34181cdc674222515e5b6
Normal-2779.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2779.png,Normal/images/Normal-2779.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2779.png,Normal/masks/Normal-2779.png,True,False,299,299,L,109.69,bff5db80a9bfd48b4579c0d06843d22b
Normal-278.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-278.png,Normal/images/Normal-278.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-278.png,Normal/masks/Normal-278.png,True,False,299,299,L,114.08,62a66ea3e1d9d91210cf3e8f9da54d1b
Normal-2780.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2780.png,Normal/images/Normal-2780.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2780.png,Normal/masks/Normal-2780.png,True,False,299,299,L,96.7,a0e07033aa71ed4cc2f58c1a86466740
Normal-2781.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2781.png,Normal/images/Normal-2781.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2781.png,Normal/masks/Normal-2781.png,True,False,299,299,L,108.9,795edf9e4c74008841b4eae3d9067952
Normal-2782.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2782.png,Normal/images/Normal-2782.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2782.png,Normal/masks/Normal-2782.png,True,False,299,299,L,120.44,24657e84522afb74dd10d7e528d43e4b
Normal-2783.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2783.png,Normal/images/Normal-2783.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2783.png,Normal/masks/Normal-2783.png,True,False,299,299,L,171.71,4d36138d638a424d02b004426dbd6fbe
Normal-2784.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2784.png,Normal/images/Normal-2784.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2784.png,Normal/masks/Normal-2784.png,True,False,299,299,L,110.08,66ba99f8d13c9f1f5f55ac7bd36a521b
Normal-2785.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2785.png,Normal/images/Normal-2785.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2785.png,Normal/masks/Normal-2785.png,True,False,299,299,L,123.36,65bb991f8ab8f1b03294f61ca5d0ca93
Normal-2786.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2786.png,Normal/images/Normal-2786.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2786.png,Normal/masks/Normal-2786.png,True,False,299,299,L,127.89,90f162e301cbf7cfeb427119a56bb5da
Normal-2787.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2787.png,Normal/images/Normal-2787.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2787.png,Normal/masks/Normal-2787.png,True,False,299,299,L,119.03,99ac05071d2443bae72cdd276d97fb8d
Normal-2788.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2788.png,Normal/images/Normal-2788.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2788.png,Normal/masks/Normal-2788.png,True,False,299,299,L,138.61,8069c8cab60087e959bd038eb2b93483
Normal-2789.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2789.png,Normal/images/Normal-2789.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2789.png,Normal/masks/Normal-2789.png,True,False,299,299,L,113.8,2769f61c5fe671f1724469524513e9ad
Normal-279.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-279.png,Normal/images/Normal-279.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-279.png,Normal/masks/Normal-279.png,True,False,299,299,L,137.82,5394fc512737486ae562b76a0695b6ee
Normal-2790.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2790.png,Normal/images/Normal-2790.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2790.png,Normal/masks/Normal-2790.png,True,False,299,299,L,142.58,2d78cc266230af0e1c679158d9e8b940
Normal-2791.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2791.png,Normal/images/Normal-2791.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2791.png,Normal/masks/Normal-2791.png,True,False,299,299,L,135.38,c5a8106743634c4c0af3c7963c460000
Normal-2792.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2792.png,Normal/images/Normal-2792.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2792.png,Normal/masks/Normal-2792.png,True,False,299,299,L,118.32,134f8da8c9c06386bb0cbe2aa7251e8f
Normal-2793.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2793.png,Normal/images/Normal-2793.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2793.png,Normal/masks/Normal-2793.png,True,False,299,299,L,143.29,4c9ec793c6a8f702f17ece75ab62af78
Normal-2794.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2794.png,Normal/images/Normal-2794.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2794.png,Normal/masks/Normal-2794.png,True,False,299,299,L,121.7,136418072350d2ef4f7ee9ffffa54224
Normal-2795.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2795.png,Normal/images/Normal-2795.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2795.png,Normal/masks/Normal-2795.png,True,False,299,299,L,122.82,50159742060bcee628196dec94a02f6d
Normal-2796.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2796.png,Normal/images/Normal-2796.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2796.png,Normal/masks/Normal-2796.png,True,False,299,299,L,160.2,03855d5cd7fab190b56e4b8a2409293f
Normal-2797.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2797.png,Normal/images/Normal-2797.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2797.png,Normal/masks/Normal-2797.png,True,False,299,299,L,163.7,ba8ef6b76a44d4f7a0a3560997fdf8fe
Normal-2798.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2798.png,Normal/images/Normal-2798.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2798.png,Normal/masks/Normal-2798.png,True,False,299,299,L,106.55,8d160a8b87769e27ce01cceb27333bac
Normal-2799.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2799.png,Normal/images/Normal-2799.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2799.png,Normal/masks/Normal-2799.png,True,False,299,299,L,144.13,a60a55eed21fb89cccfb3381b4c80211
Normal-28.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-28.png,Normal/images/Normal-28.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-28.png,Normal/masks/Normal-28.png,True,False,299,299,L,121.84,92de83da604b021c2688253be1a0339c
Normal-280.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-280.png,Normal/images/Normal-280.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-280.png,Normal/masks/Normal-280.png,True,False,299,299,L,119.28,a54b148c6f4751ccbe9645b4477a9eb2
Normal-2800.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2800.png,Normal/images/Normal-2800.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2800.png,Normal/masks/Normal-2800.png,True,False,299,299,L,125.64,b84c5ec38993195dee9f3931d59cacc8
Normal-2801.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2801.png,Normal/images/Normal-2801.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2801.png,Normal/masks/Normal-2801.png,True,False,299,299,L,144.16,36c8c423fc43d4dc1c5e202e8f473675
Normal-2802.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2802.png,Normal/images/Normal-2802.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2802.png,Normal/masks/Normal-2802.png,True,False,299,299,L,114.59,0cc5881081789984d26ef375fd94de55
Normal-2803.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2803.png,Normal/images/Normal-2803.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2803.png,Normal/masks/Normal-2803.png,True,False,299,299,L,145.77,293d598d8b64f40e09daba9dc9f4b56d
Normal-2804.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2804.png,Normal/images/Normal-2804.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2804.png,Normal/masks/Normal-2804.png,True,False,299,299,L,123.97,dfda35effc9bf2b2a5829cffe9a26cfa
Normal-2805.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2805.png,Normal/images/Normal-2805.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2805.png,Normal/masks/Normal-2805.png,True,False,299,299,L,147.37,aa1608606f0ffae75d8cdff78166ebe3
Normal-2806.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2806.png,Normal/images/Normal-2806.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2806.png,Normal/masks/Normal-2806.png,True,False,299,299,L,100.65,3982f0c38ae91b24e2c9ea27a0b89b2f
Normal-2807.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2807.png,Normal/images/Normal-2807.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2807.png,Normal/masks/Normal-2807.png,True,False,299,299,L,189.37,db111cd0026c70a1dd5acc5da13b917e
Normal-2808.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2808.png,Normal/images/Normal-2808.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2808.png,Normal/masks/Normal-2808.png,True,False,299,299,L,110.06,9e6c6d6be23976431e48626fbf71bc75
Normal-2809.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2809.png,Normal/images/Normal-2809.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2809.png,Normal/masks/Normal-2809.png,True,False,299,299,L,179.42,4acd5f93f0a81ef4da293cfa1149e233
Normal-281.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-281.png,Normal/images/Normal-281.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-281.png,Normal/masks/Normal-281.png,True,False,299,299,L,107.19,3ae42add9618781c545bef9aababaf0f
Normal-2810.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2810.png,Normal/images/Normal-2810.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2810.png,Normal/masks/Normal-2810.png,True,False,299,299,L,135.51,a065dc9bdc0dc2d57060983246317674
Normal-2811.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2811.png,Normal/images/Normal-2811.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2811.png,Normal/masks/Normal-2811.png,True,False,299,299,L,108.1,5e829d564d3f506f1cd7f208f968f017
Normal-2812.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2812.png,Normal/images/Normal-2812.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2812.png,Normal/masks/Normal-2812.png,True,False,299,299,L,162.98,e2573d7231f0853003aa9aaec22624fd
Normal-2813.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2813.png,Normal/images/Normal-2813.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2813.png,Normal/masks/Normal-2813.png,True,False,299,299,L,144.29,e6f2808389b1946aa440cf9e84145eec
Normal-2814.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2814.png,Normal/images/Normal-2814.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2814.png,Normal/masks/Normal-2814.png,True,False,299,299,L,110.77,53a3efb968908fa61bdb4522d2e45865
Normal-2815.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2815.png,Normal/images/Normal-2815.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2815.png,Normal/masks/Normal-2815.png,True,False,299,299,L,156.79,6e874af46a0596d17e467e0d636a1bd1
Normal-2816.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2816.png,Normal/images/Normal-2816.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2816.png,Normal/masks/Normal-2816.png,True,False,299,299,L,121.85,1ecc27228a5e46ac6ab07ec428d35ff8
Normal-2817.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2817.png,Normal/images/Normal-2817.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2817.png,Normal/masks/Normal-2817.png,True,False,299,299,L,125.38,95f09dc16125f58f577460536852f8d1
Normal-2818.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2818.png,Normal/images/Normal-2818.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2818.png,Normal/masks/Normal-2818.png,True,False,299,299,L,106.69,cd43efdf9d6b4d7a4a0de6a377413596
Normal-2819.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2819.png,Normal/images/Normal-2819.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2819.png,Normal/masks/Normal-2819.png,True,False,299,299,L,121.55,9265ff7ea0dd030f25da28036f27768e
Normal-282.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-282.png,Normal/images/Normal-282.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-282.png,Normal/masks/Normal-282.png,True,False,299,299,L,126.13,3bbbaa1d53d67c68b737395c25ecdf98
Normal-2820.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2820.png,Normal/images/Normal-2820.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2820.png,Normal/masks/Normal-2820.png,True,False,299,299,L,134.68,4e24a86a00e81b62202323e84516a42f
Normal-2821.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2821.png,Normal/images/Normal-2821.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2821.png,Normal/masks/Normal-2821.png,True,False,299,299,L,160.75,85d74f33df83b86e82b73c730e8ecc2b
Normal-2822.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2822.png,Normal/images/Normal-2822.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2822.png,Normal/masks/Normal-2822.png,True,False,299,299,L,129.83,b6fb30eea4b56d7027b3ddc6f7961894
Normal-2823.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2823.png,Normal/images/Normal-2823.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2823.png,Normal/masks/Normal-2823.png,True,False,299,299,L,104.81,1bf28cdee96bc1b43259567ca23340ea
Normal-2824.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2824.png,Normal/images/Normal-2824.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2824.png,Normal/masks/Normal-2824.png,True,False,299,299,L,114.18,8676631ecaf8079d90db30dad463fcbc
Normal-2825.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2825.png,Normal/images/Normal-2825.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2825.png,Normal/masks/Normal-2825.png,True,False,299,299,L,125.03,5c69f07c69664cb6f223bf8fcdf23a39
Normal-2826.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2826.png,Normal/images/Normal-2826.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2826.png,Normal/masks/Normal-2826.png,True,False,299,299,L,143.24,2a3059aa9b254a36cfc3874747baa9bf
Normal-2827.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2827.png,Normal/images/Normal-2827.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2827.png,Normal/masks/Normal-2827.png,True,False,299,299,L,181.93,7fd95228a277af269b97250d505583e3
Normal-2828.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2828.png,Normal/images/Normal-2828.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2828.png,Normal/masks/Normal-2828.png,True,False,299,299,L,126.56,acc2e7031dc699ddb28d65c29bfaecee
Normal-2829.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2829.png,Normal/images/Normal-2829.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2829.png,Normal/masks/Normal-2829.png,True,False,299,299,L,163.74,58675c28052c9e71e002573360ca975a
Normal-283.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-283.png,Normal/images/Normal-283.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-283.png,Normal/masks/Normal-283.png,True,False,299,299,L,126.61,c80685842374ddeb7c56a20731ddd467
Normal-2830.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2830.png,Normal/images/Normal-2830.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2830.png,Normal/masks/Normal-2830.png,True,False,299,299,L,120.87,941cb95ab72298cb58e42cfa66914a56
Normal-2831.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2831.png,Normal/images/Normal-2831.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2831.png,Normal/masks/Normal-2831.png,True,False,299,299,L,122.42,7249763349569bb558a526d8e75331ef
Normal-2832.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2832.png,Normal/images/Normal-2832.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2832.png,Normal/masks/Normal-2832.png,True,False,299,299,L,119.34,3945539228963abfa1349660282bb65d
Normal-2833.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2833.png,Normal/images/Normal-2833.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2833.png,Normal/masks/Normal-2833.png,True,False,299,299,L,138.51,d2506b5a949fa03525e2ac440aa6907e
Normal-2834.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2834.png,Normal/images/Normal-2834.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2834.png,Normal/masks/Normal-2834.png,True,False,299,299,L,161.79,fd8094732a620d74dad386f898afceb9
Normal-2835.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2835.png,Normal/images/Normal-2835.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2835.png,Normal/masks/Normal-2835.png,True,False,299,299,L,129.65,40f19accdc0a65a230207d3896e868a9
Normal-2836.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2836.png,Normal/images/Normal-2836.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2836.png,Normal/masks/Normal-2836.png,True,False,299,299,L,144.15,2d33f4ba71f37d3f4e62b0d64f6f4007
Normal-2837.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2837.png,Normal/images/Normal-2837.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2837.png,Normal/masks/Normal-2837.png,True,False,299,299,L,145.55,fe91833a3fd0b25a056b60db15b16251
Normal-2838.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2838.png,Normal/images/Normal-2838.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2838.png,Normal/masks/Normal-2838.png,True,False,299,299,L,140.78,e9a9f0f0039956411bdc875ec71a9a16
Normal-2839.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2839.png,Normal/images/Normal-2839.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2839.png,Normal/masks/Normal-2839.png,True,False,299,299,L,139.46,9fcd16c6f6f9149cccfe90498162e869
Normal-284.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-284.png,Normal/images/Normal-284.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-284.png,Normal/masks/Normal-284.png,True,False,299,299,L,106.37,129f19671855eca03b651f52c8db6389
Normal-2840.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2840.png,Normal/images/Normal-2840.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2840.png,Normal/masks/Normal-2840.png,True,False,299,299,L,120.08,62d44a9e5442513b66dbd0c412e1b4ac
Normal-2841.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2841.png,Normal/images/Normal-2841.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2841.png,Normal/masks/Normal-2841.png,True,False,299,299,L,118.07,58712bf7e2a8d2ed01ed9a33f0b18723
Normal-2842.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2842.png,Normal/images/Normal-2842.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2842.png,Normal/masks/Normal-2842.png,True,False,299,299,L,137.99,e39b400e09b24d9db70b29726fb849b6
Normal-2843.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2843.png,Normal/images/Normal-2843.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2843.png,Normal/masks/Normal-2843.png,True,False,299,299,L,111.2,d68700840cd929aa93f9f51582715875
Normal-2844.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2844.png,Normal/images/Normal-2844.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2844.png,Normal/masks/Normal-2844.png,True,False,299,299,L,104.31,327a3846cd9493caf668dd0d56f6798c
Normal-2845.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2845.png,Normal/images/Normal-2845.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2845.png,Normal/masks/Normal-2845.png,True,False,299,299,L,107.65,7898504329c09e6663758d9c58fe388d
Normal-2846.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2846.png,Normal/images/Normal-2846.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2846.png,Normal/masks/Normal-2846.png,True,False,299,299,L,160.22,69ff1e6a5e84a5c0356970633e81ca52
Normal-2847.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2847.png,Normal/images/Normal-2847.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2847.png,Normal/masks/Normal-2847.png,True,False,299,299,L,130.35,7ee8d982728a5670c68c5e0198f9bc16
Normal-2848.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2848.png,Normal/images/Normal-2848.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2848.png,Normal/masks/Normal-2848.png,True,False,299,299,L,111.39,4a94a8ffe6cc783a0b46bd4edfb4df19
Normal-2849.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2849.png,Normal/images/Normal-2849.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2849.png,Normal/masks/Normal-2849.png,True,False,299,299,L,103.12,503d33ddf9871dd0b5661be0d732ebdd
Normal-285.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-285.png,Normal/images/Normal-285.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-285.png,Normal/masks/Normal-285.png,True,False,299,299,L,134.0,1001342fe30cf320016dc8da71b41585
Normal-2850.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2850.png,Normal/images/Normal-2850.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2850.png,Normal/masks/Normal-2850.png,True,False,299,299,L,113.73,ba33e5beb04cedb2293712f52316f815
Normal-2851.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2851.png,Normal/images/Normal-2851.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2851.png,Normal/masks/Normal-2851.png,True,False,299,299,L,115.75,fdcfd79a107df1e8fe936c2b14547b04
Normal-2852.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2852.png,Normal/images/Normal-2852.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2852.png,Normal/masks/Normal-2852.png,True,False,299,299,L,104.42,66a50a3b3b8796aa326e86c54d341cab
Normal-2853.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2853.png,Normal/images/Normal-2853.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2853.png,Normal/masks/Normal-2853.png,True,False,299,299,L,121.94,262b55de0838f061c79b69074e8c1dfb
Normal-2854.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2854.png,Normal/images/Normal-2854.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2854.png,Normal/masks/Normal-2854.png,True,False,299,299,L,105.55,0840a178580d6d4a8d07e3acf52fba6d
Normal-2855.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2855.png,Normal/images/Normal-2855.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2855.png,Normal/masks/Normal-2855.png,True,False,299,299,L,115.15,c4112ed71d8226483b9ff6dbce599a50
Normal-2856.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2856.png,Normal/images/Normal-2856.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2856.png,Normal/masks/Normal-2856.png,True,False,299,299,L,101.69,f9f54209ff8af42e705e37a23c83b5e2
Normal-2857.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2857.png,Normal/images/Normal-2857.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2857.png,Normal/masks/Normal-2857.png,True,False,299,299,L,140.64,89fb08867d25e965a255aa5660d9a4ea
Normal-2858.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2858.png,Normal/images/Normal-2858.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2858.png,Normal/masks/Normal-2858.png,True,False,299,299,L,172.03,608bfb1459afd2a807e47cb5b86c7b0f
Normal-2859.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2859.png,Normal/images/Normal-2859.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2859.png,Normal/masks/Normal-2859.png,True,False,299,299,L,118.14,45005914f41501707ad7148cd584b2d7
Normal-286.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-286.png,Normal/images/Normal-286.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-286.png,Normal/masks/Normal-286.png,True,False,299,299,L,147.67,d4cab32df4d00361fd7af42e2d101cbc
Normal-2860.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2860.png,Normal/images/Normal-2860.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2860.png,Normal/masks/Normal-2860.png,True,False,299,299,L,131.85,2ac62c02d855a850bff83ece047f16f7
Normal-2861.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2861.png,Normal/images/Normal-2861.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2861.png,Normal/masks/Normal-2861.png,True,False,299,299,L,108.99,2357802967cb1206d5e94a409dac52a0
Normal-2862.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2862.png,Normal/images/Normal-2862.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2862.png,Normal/masks/Normal-2862.png,True,False,299,299,L,91.07,69c3ff3a7a7c07486a09fda7e3e6a0a9
Normal-2863.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2863.png,Normal/images/Normal-2863.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2863.png,Normal/masks/Normal-2863.png,True,False,299,299,L,160.66,e14200c16ac102f2b1382497dbce836a
Normal-2864.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2864.png,Normal/images/Normal-2864.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2864.png,Normal/masks/Normal-2864.png,True,False,299,299,L,98.15,3c6eae0dc3cfe20986ac5ffa793ab6e8
Normal-2865.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2865.png,Normal/images/Normal-2865.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2865.png,Normal/masks/Normal-2865.png,True,False,299,299,L,123.17,60cb8b82dda474ccfea64f0a1125185c
Normal-2866.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2866.png,Normal/images/Normal-2866.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2866.png,Normal/masks/Normal-2866.png,True,False,299,299,L,139.56,a7f3eda50306b4017109045216e8e334
Normal-2867.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2867.png,Normal/images/Normal-2867.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2867.png,Normal/masks/Normal-2867.png,True,False,299,299,L,88.55,00b23273410ea1c07800a435e65b8f15
Normal-2868.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2868.png,Normal/images/Normal-2868.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2868.png,Normal/masks/Normal-2868.png,True,False,299,299,L,97.81,9d3f68ee8a75563f289f54f50ff746b9
Normal-2869.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2869.png,Normal/images/Normal-2869.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2869.png,Normal/masks/Normal-2869.png,True,False,299,299,L,133.51,0a1fb58ccc155aa8578b799bf0ae0e79
Normal-287.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-287.png,Normal/images/Normal-287.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-287.png,Normal/masks/Normal-287.png,True,False,299,299,L,113.47,6e5ab13a629c41808f4d16016d330d14
Normal-2870.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2870.png,Normal/images/Normal-2870.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2870.png,Normal/masks/Normal-2870.png,True,False,299,299,L,169.71,dbd1e5053577127bd863f3e1ede79649
Normal-2871.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2871.png,Normal/images/Normal-2871.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2871.png,Normal/masks/Normal-2871.png,True,False,299,299,L,130.12,9e6b195be2af5b8bde7d67d61904fd91
Normal-2872.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2872.png,Normal/images/Normal-2872.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2872.png,Normal/masks/Normal-2872.png,True,False,299,299,L,110.56,9b57de9110873d95a21707be43b2b64b
Normal-2873.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2873.png,Normal/images/Normal-2873.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2873.png,Normal/masks/Normal-2873.png,True,False,299,299,L,121.29,359d35699781163ce7e0bfb9df7e25fd
Normal-2874.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2874.png,Normal/images/Normal-2874.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2874.png,Normal/masks/Normal-2874.png,True,False,299,299,L,106.59,e0c6c4c13be4cd79da0bee05927beae7
Normal-2875.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2875.png,Normal/images/Normal-2875.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2875.png,Normal/masks/Normal-2875.png,True,False,299,299,L,104.53,0312ef8e6b72345f3db9bf2e45a0791a
Normal-2876.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2876.png,Normal/images/Normal-2876.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2876.png,Normal/masks/Normal-2876.png,True,False,299,299,L,113.1,67b873ed700f88ffdc7046c81b0d1539
Normal-2877.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2877.png,Normal/images/Normal-2877.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2877.png,Normal/masks/Normal-2877.png,True,False,299,299,L,131.01,2e33557b6dd6e4821b27942d25202530
Normal-2878.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2878.png,Normal/images/Normal-2878.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2878.png,Normal/masks/Normal-2878.png,True,False,299,299,L,111.21,95b18d0a13ec97f2c61cd39c16437969
Normal-2879.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2879.png,Normal/images/Normal-2879.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2879.png,Normal/masks/Normal-2879.png,True,False,299,299,L,101.08,9f5f78a57c4474062be21586757b9baf
Normal-288.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-288.png,Normal/images/Normal-288.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-288.png,Normal/masks/Normal-288.png,True,False,299,299,L,122.99,605c3f95bf3d9e23dce5547c17822f13
Normal-2880.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2880.png,Normal/images/Normal-2880.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2880.png,Normal/masks/Normal-2880.png,True,False,299,299,L,119.62,85de0cf2bacdc6ad96684c7dbfbde407
Normal-2881.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2881.png,Normal/images/Normal-2881.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2881.png,Normal/masks/Normal-2881.png,True,False,299,299,L,41.36,fae3d5d2a7f174f182b554baef76613d
Normal-2882.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2882.png,Normal/images/Normal-2882.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2882.png,Normal/masks/Normal-2882.png,True,False,299,299,L,160.04,09060addd1367ff80bef1c139369885d
Normal-2883.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2883.png,Normal/images/Normal-2883.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2883.png,Normal/masks/Normal-2883.png,True,False,299,299,L,114.05,2ed1f35e93c864c3b7a85cdb2da312df
Normal-2884.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2884.png,Normal/images/Normal-2884.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2884.png,Normal/masks/Normal-2884.png,True,False,299,299,L,176.79,5a81a7da35572c08c9e6ef700bce89ca
Normal-2885.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2885.png,Normal/images/Normal-2885.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2885.png,Normal/masks/Normal-2885.png,True,False,299,299,L,174.13,f7f72992d7754b72d901d561ac04a8b3
Normal-2886.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2886.png,Normal/images/Normal-2886.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2886.png,Normal/masks/Normal-2886.png,True,False,299,299,L,128.36,a5100d7e19670a7d9f0867e8f20e61cd
Normal-2887.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2887.png,Normal/images/Normal-2887.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2887.png,Normal/masks/Normal-2887.png,True,False,299,299,L,98.39,d8afa2ac4fe0855b1c62aa8cde207092
Normal-2888.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2888.png,Normal/images/Normal-2888.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2888.png,Normal/masks/Normal-2888.png,True,False,299,299,L,155.32,18b353da2ba54f9dd2df4cfc8a023758
Normal-2889.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2889.png,Normal/images/Normal-2889.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2889.png,Normal/masks/Normal-2889.png,True,False,299,299,L,161.53,22d3577855ea2e2d5e81dbd53905cc5b
Normal-289.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-289.png,Normal/images/Normal-289.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-289.png,Normal/masks/Normal-289.png,True,False,299,299,L,120.38,7efce324b9d73c2ad9c7fb469b9e8a81
Normal-2890.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2890.png,Normal/images/Normal-2890.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2890.png,Normal/masks/Normal-2890.png,True,False,299,299,L,170.08,f12607c27a7652aef5f9e1c6acd10a06
Normal-2891.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2891.png,Normal/images/Normal-2891.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2891.png,Normal/masks/Normal-2891.png,True,False,299,299,L,117.31,f742059ab02bea97a2f9007253b050c8
Normal-2892.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2892.png,Normal/images/Normal-2892.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2892.png,Normal/masks/Normal-2892.png,True,False,299,299,L,138.77,62b1c0eac65eba7aa7d7e4a6d3f24fc2
Normal-2893.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2893.png,Normal/images/Normal-2893.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2893.png,Normal/masks/Normal-2893.png,True,False,299,299,L,155.76,0f4993300d68cb1e0fa26f68ead1eccd
Normal-2894.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2894.png,Normal/images/Normal-2894.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2894.png,Normal/masks/Normal-2894.png,True,False,299,299,L,181.13,4d1ae7284ae5f364a547ebb0848f19f4
Normal-2895.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2895.png,Normal/images/Normal-2895.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2895.png,Normal/masks/Normal-2895.png,True,False,299,299,L,111.93,a4f03838b208bbe15f46a6060a4cc255
Normal-2896.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2896.png,Normal/images/Normal-2896.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2896.png,Normal/masks/Normal-2896.png,True,False,299,299,L,100.21,b82b5475562dc92c503cc756248fd2bc
Normal-2897.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2897.png,Normal/images/Normal-2897.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2897.png,Normal/masks/Normal-2897.png,True,False,299,299,L,175.74,7731ecc2a42149692c360531a9cdb90b
Normal-2898.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2898.png,Normal/images/Normal-2898.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2898.png,Normal/masks/Normal-2898.png,True,False,299,299,L,169.01,82e9ea99137e6ffa2fa7a237003c20fe
Normal-2899.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2899.png,Normal/images/Normal-2899.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2899.png,Normal/masks/Normal-2899.png,True,False,299,299,L,130.06,19145f0e87480ea42875cfc6dc1a98e6
Normal-29.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-29.png,Normal/images/Normal-29.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-29.png,Normal/masks/Normal-29.png,True,False,299,299,L,172.44,54414f2cdacc30a2f12832045c338361
Normal-290.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-290.png,Normal/images/Normal-290.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-290.png,Normal/masks/Normal-290.png,True,False,299,299,L,126.76,6d34fa3d108428cc27e2aabc449d892e
Normal-2900.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2900.png,Normal/images/Normal-2900.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2900.png,Normal/masks/Normal-2900.png,True,False,299,299,L,122.61,480a1626831e88e764881dfef0a13255
Normal-2901.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2901.png,Normal/images/Normal-2901.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2901.png,Normal/masks/Normal-2901.png,True,False,299,299,L,126.42,1c68bad3f8fd0c02268c92ed3a81cb3b
Normal-2902.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2902.png,Normal/images/Normal-2902.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2902.png,Normal/masks/Normal-2902.png,True,False,299,299,L,166.13,297d113da8d1f5202c14436450327fcc
Normal-2903.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2903.png,Normal/images/Normal-2903.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2903.png,Normal/masks/Normal-2903.png,True,False,299,299,L,105.05,2d65b4a695b8cc9c5ecd93a54a693fa3
Normal-2904.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2904.png,Normal/images/Normal-2904.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2904.png,Normal/masks/Normal-2904.png,True,False,299,299,L,164.29,c4353dd619fb5d4393cc10c6de3f3d84
Normal-2905.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2905.png,Normal/images/Normal-2905.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2905.png,Normal/masks/Normal-2905.png,True,False,299,299,L,147.23,1d8c7b9ff008c335c221658c457b05e4
Normal-2906.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2906.png,Normal/images/Normal-2906.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2906.png,Normal/masks/Normal-2906.png,True,False,299,299,L,113.58,8624359c6f36a8fb013a4ef38cb23436
Normal-2907.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2907.png,Normal/images/Normal-2907.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2907.png,Normal/masks/Normal-2907.png,True,False,299,299,L,101.52,2381fb90028ce1f172684cefd14a1d94
Normal-2908.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2908.png,Normal/images/Normal-2908.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2908.png,Normal/masks/Normal-2908.png,True,False,299,299,L,169.31,60e86f56fa785b013039c68377cd4f45
Normal-2909.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2909.png,Normal/images/Normal-2909.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2909.png,Normal/masks/Normal-2909.png,True,False,299,299,L,147.91,f632b1d7da2eac3f7dc4b6fd428eaa2f
Normal-291.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-291.png,Normal/images/Normal-291.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-291.png,Normal/masks/Normal-291.png,True,False,299,299,L,125.86,c154032eb93f188e3c4dbd4c57f90675
Normal-2910.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2910.png,Normal/images/Normal-2910.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2910.png,Normal/masks/Normal-2910.png,True,False,299,299,L,113.47,46b42edd2933675ad87f5228707f02fc
Normal-2911.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2911.png,Normal/images/Normal-2911.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2911.png,Normal/masks/Normal-2911.png,True,False,299,299,L,143.71,e9ad954c34aa7e91ad32ea54ea42664a
Normal-2912.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2912.png,Normal/images/Normal-2912.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2912.png,Normal/masks/Normal-2912.png,True,False,299,299,L,120.23,799c1af800c4f138fca007ae71777b60
Normal-2913.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2913.png,Normal/images/Normal-2913.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2913.png,Normal/masks/Normal-2913.png,True,False,299,299,L,116.61,56384c37c6d6bb5b7e517013c2ebbc0b
Normal-2914.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2914.png,Normal/images/Normal-2914.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2914.png,Normal/masks/Normal-2914.png,True,False,299,299,L,145.04,f4dd1b4f4ce4619301a00681f0fd5525
Normal-2915.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2915.png,Normal/images/Normal-2915.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2915.png,Normal/masks/Normal-2915.png,True,False,299,299,L,173.06,d623be7a1ead1911a5f327c05cb22020
Normal-2916.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2916.png,Normal/images/Normal-2916.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2916.png,Normal/masks/Normal-2916.png,True,False,299,299,L,129.06,b499171a73c4951a2224050e68ef7675
Normal-2917.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2917.png,Normal/images/Normal-2917.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2917.png,Normal/masks/Normal-2917.png,True,False,299,299,L,134.84,3bf58850548ee0fa8023a598cef67f43
Normal-2918.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2918.png,Normal/images/Normal-2918.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2918.png,Normal/masks/Normal-2918.png,True,False,299,299,L,106.01,4af2f36a04ea619ecadd8ca40192ab2e
Normal-2919.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2919.png,Normal/images/Normal-2919.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2919.png,Normal/masks/Normal-2919.png,True,False,299,299,L,168.56,65ca89aa1161b3b1c928c36ecf5f1891
Normal-292.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-292.png,Normal/images/Normal-292.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-292.png,Normal/masks/Normal-292.png,True,False,299,299,L,135.97,b4ee4d7a0cc4f0d2cf5671aeff0b2d55
Normal-2920.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2920.png,Normal/images/Normal-2920.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2920.png,Normal/masks/Normal-2920.png,True,False,299,299,L,117.77,5d1d628e5a9ccc83f73a39abef2111fa
Normal-2921.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2921.png,Normal/images/Normal-2921.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2921.png,Normal/masks/Normal-2921.png,True,False,299,299,L,135.65,69929ea06051ac38e15fc99bdf141cfe
Normal-2922.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2922.png,Normal/images/Normal-2922.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2922.png,Normal/masks/Normal-2922.png,True,False,299,299,L,131.08,dc8b7fc0d873ee04917a18c287247f90
Normal-2923.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2923.png,Normal/images/Normal-2923.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2923.png,Normal/masks/Normal-2923.png,True,False,299,299,L,172.58,daf05677f70c7fe9140a434eaff8067b
Normal-2924.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2924.png,Normal/images/Normal-2924.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2924.png,Normal/masks/Normal-2924.png,True,False,299,299,L,116.34,6a2c865c5a55d7b7df62a1c670f410f0
Normal-2925.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2925.png,Normal/images/Normal-2925.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2925.png,Normal/masks/Normal-2925.png,True,False,299,299,L,153.63,0bfb4352d4625d6d5f686466d8a20f03
Normal-2926.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2926.png,Normal/images/Normal-2926.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2926.png,Normal/masks/Normal-2926.png,True,False,299,299,L,99.32,ad2d8d519304e26122def34e06c2eafd
Normal-2927.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2927.png,Normal/images/Normal-2927.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2927.png,Normal/masks/Normal-2927.png,True,False,299,299,L,165.84,b1bb3d6734cdb403ca0b8bc11cc9556a
Normal-2928.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2928.png,Normal/images/Normal-2928.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2928.png,Normal/masks/Normal-2928.png,True,False,299,299,L,114.89,be08ad1412535bcc252d6a90a6a35a64
Normal-2929.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2929.png,Normal/images/Normal-2929.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2929.png,Normal/masks/Normal-2929.png,True,False,299,299,L,105.7,ee3c5714223ba6cecdeb2ef665da0dbb
Normal-293.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-293.png,Normal/images/Normal-293.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-293.png,Normal/masks/Normal-293.png,True,False,299,299,L,108.29,b1c9c7fa300bdf26193db8310db1c6ee
Normal-2930.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2930.png,Normal/images/Normal-2930.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2930.png,Normal/masks/Normal-2930.png,True,False,299,299,L,125.97,b8ea5c1d82460c6f5bf0226ab04548fd
Normal-2931.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2931.png,Normal/images/Normal-2931.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2931.png,Normal/masks/Normal-2931.png,True,False,299,299,L,157.68,f7cca0c07193f88921dd13a21747312d
Normal-2932.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2932.png,Normal/images/Normal-2932.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2932.png,Normal/masks/Normal-2932.png,True,False,299,299,L,127.75,b06c2003a7748b5be3783af423063b0e
Normal-2933.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2933.png,Normal/images/Normal-2933.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2933.png,Normal/masks/Normal-2933.png,True,False,299,299,L,180.42,419d6ad68e5e203d19d861027b0e5716
Normal-2934.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2934.png,Normal/images/Normal-2934.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2934.png,Normal/masks/Normal-2934.png,True,False,299,299,L,106.82,1d794ed474c84e3f409e095d02697951
Normal-2935.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2935.png,Normal/images/Normal-2935.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2935.png,Normal/masks/Normal-2935.png,True,False,299,299,L,108.24,7d09f2fbc5e13d63dfcb6868917fd198
Normal-2936.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2936.png,Normal/images/Normal-2936.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2936.png,Normal/masks/Normal-2936.png,True,False,299,299,L,164.37,f530a15ec0df8df76878bbd7c7cc9ecc
Normal-2937.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2937.png,Normal/images/Normal-2937.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2937.png,Normal/masks/Normal-2937.png,True,False,299,299,L,106.65,48d31bb4280890ca891944d1ec58944a
Normal-2938.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2938.png,Normal/images/Normal-2938.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2938.png,Normal/masks/Normal-2938.png,True,False,299,299,L,174.8,45288d0575b61fb330e1d5e2f9359d13
Normal-2939.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2939.png,Normal/images/Normal-2939.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2939.png,Normal/masks/Normal-2939.png,True,False,299,299,L,163.72,05c5fe3d03b6b5effc36e345e5e5382e
Normal-294.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-294.png,Normal/images/Normal-294.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-294.png,Normal/masks/Normal-294.png,True,False,299,299,L,134.31,05ac0a8e4b7c000e3d52c1bc516a8c34
Normal-2940.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2940.png,Normal/images/Normal-2940.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2940.png,Normal/masks/Normal-2940.png,True,False,299,299,L,108.63,9d4c17011ac7517981914225e251d9a8
Normal-2941.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2941.png,Normal/images/Normal-2941.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2941.png,Normal/masks/Normal-2941.png,True,False,299,299,L,115.19,a5c000f345eaa9b15b9ef40967878eeb
Normal-2942.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2942.png,Normal/images/Normal-2942.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2942.png,Normal/masks/Normal-2942.png,True,False,299,299,L,106.33,a67bb91a05385f068a662409110f4fea
Normal-2943.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2943.png,Normal/images/Normal-2943.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2943.png,Normal/masks/Normal-2943.png,True,False,299,299,L,169.8,8745eb80fe8d1d002357c41721453537
Normal-2944.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2944.png,Normal/images/Normal-2944.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2944.png,Normal/masks/Normal-2944.png,True,False,299,299,L,109.65,08c480851c79ec6b1e5583dd53f3eb50
Normal-2945.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2945.png,Normal/images/Normal-2945.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2945.png,Normal/masks/Normal-2945.png,True,False,299,299,L,105.95,567aab0d8b97d2f3d9aebba279f4c1b7
Normal-2946.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2946.png,Normal/images/Normal-2946.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2946.png,Normal/masks/Normal-2946.png,True,False,299,299,L,110.53,62ab78948a807fe097d5899f9e2a65e5
Normal-2947.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2947.png,Normal/images/Normal-2947.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2947.png,Normal/masks/Normal-2947.png,True,False,299,299,L,122.8,f8f091ce7b3eea7b838bf512acfb35ff
Normal-2948.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2948.png,Normal/images/Normal-2948.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2948.png,Normal/masks/Normal-2948.png,True,False,299,299,L,54.03,c7bf45df13cea8887661b09a651ecee0
Normal-2949.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2949.png,Normal/images/Normal-2949.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2949.png,Normal/masks/Normal-2949.png,True,False,299,299,L,161.79,d28bf813a6f0d989e12e17646e18b049
Normal-295.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-295.png,Normal/images/Normal-295.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-295.png,Normal/masks/Normal-295.png,True,False,299,299,L,125.34,34727b6ac50ce7d13672d72c53735811
Normal-2950.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2950.png,Normal/images/Normal-2950.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2950.png,Normal/masks/Normal-2950.png,True,False,299,299,L,174.14,4e6cd8461d3baa6c3ff06e62e8bb7b4e
Normal-2951.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2951.png,Normal/images/Normal-2951.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2951.png,Normal/masks/Normal-2951.png,True,False,299,299,L,139.23,c78c49387776c242489119d9246a8a63
Normal-2952.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2952.png,Normal/images/Normal-2952.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2952.png,Normal/masks/Normal-2952.png,True,False,299,299,L,131.32,3d4d091bbd6993c8e6839be0238f4eef
Normal-2953.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2953.png,Normal/images/Normal-2953.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2953.png,Normal/masks/Normal-2953.png,True,False,299,299,L,161.77,184865848d92569d0d302e93d5283e81
Normal-2954.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2954.png,Normal/images/Normal-2954.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2954.png,Normal/masks/Normal-2954.png,True,False,299,299,L,154.2,41c36d4814b49fbe26808b1013ff8222
Normal-2955.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2955.png,Normal/images/Normal-2955.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2955.png,Normal/masks/Normal-2955.png,True,False,299,299,L,138.61,55739b5b6739ca78181c4f6bf1eed93d
Normal-2956.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2956.png,Normal/images/Normal-2956.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2956.png,Normal/masks/Normal-2956.png,True,False,299,299,L,106.49,4cba89af145db64f93b01b2cca45b197
Normal-2957.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2957.png,Normal/images/Normal-2957.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2957.png,Normal/masks/Normal-2957.png,True,False,299,299,L,181.28,5bc541b534f78c368826fd93fafcbffa
Normal-2958.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2958.png,Normal/images/Normal-2958.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2958.png,Normal/masks/Normal-2958.png,True,False,299,299,L,164.81,377ac94a2376554d61d07f344537c26d
Normal-2959.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2959.png,Normal/images/Normal-2959.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2959.png,Normal/masks/Normal-2959.png,True,False,299,299,L,154.13,bad0e0f790549cf28b32497cfd9e436f
Normal-296.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-296.png,Normal/images/Normal-296.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-296.png,Normal/masks/Normal-296.png,True,False,299,299,L,90.83,90801803d644f728741ba47008f9f46d
Normal-2960.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2960.png,Normal/images/Normal-2960.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2960.png,Normal/masks/Normal-2960.png,True,False,299,299,L,100.27,422e0f23e5d486d9a573afcfb537861f
Normal-2961.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2961.png,Normal/images/Normal-2961.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2961.png,Normal/masks/Normal-2961.png,True,False,299,299,L,112.12,2c0136423fda1a6159f074d8e59b87cc
Normal-2962.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2962.png,Normal/images/Normal-2962.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2962.png,Normal/masks/Normal-2962.png,True,False,299,299,L,113.34,feb25f532e5834eeeb8b9163a93ed751
Normal-2963.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2963.png,Normal/images/Normal-2963.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2963.png,Normal/masks/Normal-2963.png,True,False,299,299,L,146.46,6a2626a9d023712df0859b7005b29465
Normal-2964.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2964.png,Normal/images/Normal-2964.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2964.png,Normal/masks/Normal-2964.png,True,False,299,299,L,164.52,ae5c376b5895a89edd572e7d1e59f7a6
Normal-2965.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2965.png,Normal/images/Normal-2965.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2965.png,Normal/masks/Normal-2965.png,True,False,299,299,L,125.36,b36218daa1d190cc091bcd2e2e097c66
Normal-2966.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2966.png,Normal/images/Normal-2966.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2966.png,Normal/masks/Normal-2966.png,True,False,299,299,L,126.09,ea9af1848ed686e3520d8ecdd15e094b
Normal-2967.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2967.png,Normal/images/Normal-2967.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2967.png,Normal/masks/Normal-2967.png,True,False,299,299,L,139.94,c206eae747ad1e9394e370fbf8a63632
Normal-2968.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2968.png,Normal/images/Normal-2968.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2968.png,Normal/masks/Normal-2968.png,True,False,299,299,L,175.66,bed96fc914e7d0f8c14bb9d056404f25
Normal-2969.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2969.png,Normal/images/Normal-2969.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2969.png,Normal/masks/Normal-2969.png,True,False,299,299,L,99.0,0390848b37085ba31c5f8672b07a0d97
Normal-297.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-297.png,Normal/images/Normal-297.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-297.png,Normal/masks/Normal-297.png,True,False,299,299,L,107.15,dc1dca4d45788830e2aa2a56b48f0286
Normal-2970.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2970.png,Normal/images/Normal-2970.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2970.png,Normal/masks/Normal-2970.png,True,False,299,299,L,130.66,11d59162714329a78319881110b4e200
Normal-2971.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2971.png,Normal/images/Normal-2971.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2971.png,Normal/masks/Normal-2971.png,True,False,299,299,L,118.59,3953c670be132ad5a7673f046f6b8b78
Normal-2972.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2972.png,Normal/images/Normal-2972.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2972.png,Normal/masks/Normal-2972.png,True,False,299,299,L,109.79,b8860c39e30efa525a3d3ec4772a825c
Normal-2973.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2973.png,Normal/images/Normal-2973.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2973.png,Normal/masks/Normal-2973.png,True,False,299,299,L,132.31,8adec3abb1a07284b88033b36242ce5e
Normal-2974.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2974.png,Normal/images/Normal-2974.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2974.png,Normal/masks/Normal-2974.png,True,False,299,299,L,124.02,a45d0e2b7a93e86495ff55eb74d5f69f
Normal-2975.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2975.png,Normal/images/Normal-2975.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2975.png,Normal/masks/Normal-2975.png,True,False,299,299,L,146.35,71d2514f5d48d8cfc0244cc14e0d9c10
Normal-2976.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2976.png,Normal/images/Normal-2976.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2976.png,Normal/masks/Normal-2976.png,True,False,299,299,L,166.41,01f5c6e55edcdcc8f6f9c8d8d7fd7f10
Normal-2977.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2977.png,Normal/images/Normal-2977.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2977.png,Normal/masks/Normal-2977.png,True,False,299,299,L,137.14,cc617e8b32d1d5bd85657953acf7948b
Normal-2978.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2978.png,Normal/images/Normal-2978.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2978.png,Normal/masks/Normal-2978.png,True,False,299,299,L,106.16,97ac2ee5a104bf89334dbafb565ad7f5
Normal-2979.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2979.png,Normal/images/Normal-2979.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2979.png,Normal/masks/Normal-2979.png,True,False,299,299,L,130.75,fadec3d590fa39e0559d6ea8acc9e49c
Normal-298.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-298.png,Normal/images/Normal-298.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-298.png,Normal/masks/Normal-298.png,True,False,299,299,L,134.48,bac767b980d79ce580480361556b715a
Normal-2980.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2980.png,Normal/images/Normal-2980.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2980.png,Normal/masks/Normal-2980.png,True,False,299,299,L,176.37,8a8de6bdbdc71677b40284c838badf61
Normal-2981.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2981.png,Normal/images/Normal-2981.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2981.png,Normal/masks/Normal-2981.png,True,False,299,299,L,108.24,1690afb5c24cd5774db7d07d7a4ccd0b
Normal-2982.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2982.png,Normal/images/Normal-2982.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2982.png,Normal/masks/Normal-2982.png,True,False,299,299,L,137.47,211641650c50fb1a9198aa7aa9eca282
Normal-2983.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2983.png,Normal/images/Normal-2983.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2983.png,Normal/masks/Normal-2983.png,True,False,299,299,L,130.69,24c8960a9e4486dc6654987d56892812
Normal-2984.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2984.png,Normal/images/Normal-2984.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2984.png,Normal/masks/Normal-2984.png,True,False,299,299,L,134.33,7dc0e88cad943984671ed3365261a927
Normal-2985.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2985.png,Normal/images/Normal-2985.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2985.png,Normal/masks/Normal-2985.png,True,False,299,299,L,126.91,701ab6e3b70a377e2405d8fbd0f6a72b
Normal-2986.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2986.png,Normal/images/Normal-2986.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2986.png,Normal/masks/Normal-2986.png,True,False,299,299,L,110.2,6c5a2a395fff64010c9f36cc6b1f25cc
Normal-2987.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2987.png,Normal/images/Normal-2987.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2987.png,Normal/masks/Normal-2987.png,True,False,299,299,L,138.99,b308c7d6901313b7759f0649bdcfe728
Normal-2988.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2988.png,Normal/images/Normal-2988.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2988.png,Normal/masks/Normal-2988.png,True,False,299,299,L,129.21,7dbbfecd0ccaa14a6e1e7b77edea1da6
Normal-2989.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2989.png,Normal/images/Normal-2989.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2989.png,Normal/masks/Normal-2989.png,True,False,299,299,L,128.09,915d3e2101e7f7f7a1adc21d68bfe9a1
Normal-299.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-299.png,Normal/images/Normal-299.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-299.png,Normal/masks/Normal-299.png,True,False,299,299,L,127.63,8b18cc791af666ffc44c465e548bf8e2
Normal-2990.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2990.png,Normal/images/Normal-2990.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2990.png,Normal/masks/Normal-2990.png,True,False,299,299,L,131.04,6288159862262d59b28cf20e60a4feb2
Normal-2991.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2991.png,Normal/images/Normal-2991.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2991.png,Normal/masks/Normal-2991.png,True,False,299,299,L,113.47,97e0dc78aab35ae7d9a3999a6a3f20b5
Normal-2992.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2992.png,Normal/images/Normal-2992.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2992.png,Normal/masks/Normal-2992.png,True,False,299,299,L,168.48,18ea14beb82e5a389e1d5cdbbcc0eb00
Normal-2993.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2993.png,Normal/images/Normal-2993.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2993.png,Normal/masks/Normal-2993.png,True,False,299,299,L,122.06,078bcf8eb5f1123389315009c27de12e
Normal-2994.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2994.png,Normal/images/Normal-2994.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2994.png,Normal/masks/Normal-2994.png,True,False,299,299,L,106.14,e1b752b31213c66e2e4629318b680295
Normal-2995.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2995.png,Normal/images/Normal-2995.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2995.png,Normal/masks/Normal-2995.png,True,False,299,299,L,121.19,3ee3c740d589be8fbdbb82df9cbef141
Normal-2996.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2996.png,Normal/images/Normal-2996.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2996.png,Normal/masks/Normal-2996.png,True,False,299,299,L,146.0,820fa70330a6feda797b9e5e48d131e6
Normal-2997.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2997.png,Normal/images/Normal-2997.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2997.png,Normal/masks/Normal-2997.png,True,False,299,299,L,122.63,2579b594b6b22ba7331198c9d0e3f669
Normal-2998.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2998.png,Normal/images/Normal-2998.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2998.png,Normal/masks/Normal-2998.png,True,False,299,299,L,103.64,8c4ece4569a750faa4420050c7ebaa97
Normal-2999.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-2999.png,Normal/images/Normal-2999.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-2999.png,Normal/masks/Normal-2999.png,True,False,299,299,L,118.68,e4a6b420a5f009373768a48df1374f9a
Normal-3.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3.png,Normal/images/Normal-3.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3.png,Normal/masks/Normal-3.png,True,False,299,299,L,155.09,9a311f20f62b0aeae644f60242376963
Normal-30.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-30.png,Normal/images/Normal-30.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-30.png,Normal/masks/Normal-30.png,True,False,299,299,L,111.46,fa9fe16b2baeebb332d025f88eb60b08
Normal-300.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-300.png,Normal/images/Normal-300.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-300.png,Normal/masks/Normal-300.png,True,False,299,299,L,126.5,294ef6a673afea0e08940b384964f052
Normal-3000.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3000.png,Normal/images/Normal-3000.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3000.png,Normal/masks/Normal-3000.png,True,False,299,299,L,131.43,cdaf1b75bfdc524ce4670584e81e81a5
Normal-3001.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3001.png,Normal/images/Normal-3001.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3001.png,Normal/masks/Normal-3001.png,True,False,299,299,L,99.37,2258e8f707dc27832bb968dc13b229ca
Normal-3002.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3002.png,Normal/images/Normal-3002.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3002.png,Normal/masks/Normal-3002.png,True,False,299,299,L,118.13,8a849b5e9a940651dd776bf397441502
Normal-3003.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3003.png,Normal/images/Normal-3003.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3003.png,Normal/masks/Normal-3003.png,True,False,299,299,L,135.38,7b7ebe987ad1791698a5d3ebbc644b74
Normal-3004.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3004.png,Normal/images/Normal-3004.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3004.png,Normal/masks/Normal-3004.png,True,False,299,299,L,171.02,788677f87ed22a39f74f32b60cc86e7f
Normal-3005.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3005.png,Normal/images/Normal-3005.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3005.png,Normal/masks/Normal-3005.png,True,False,299,299,L,126.3,103c5afc40251442a86febe4f755bb09
Normal-3006.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3006.png,Normal/images/Normal-3006.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3006.png,Normal/masks/Normal-3006.png,True,False,299,299,L,132.77,f6deb7ba6991d154c3871e83ddc8232f
Normal-3007.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3007.png,Normal/images/Normal-3007.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3007.png,Normal/masks/Normal-3007.png,True,False,299,299,L,106.16,d2fc3bc7b43ce662349949095131b2e9
Normal-3008.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3008.png,Normal/images/Normal-3008.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3008.png,Normal/masks/Normal-3008.png,True,False,299,299,L,118.41,46594b334f0248f3d83d1f5bab5ddc3b
Normal-3009.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3009.png,Normal/images/Normal-3009.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3009.png,Normal/masks/Normal-3009.png,True,False,299,299,L,161.56,a1d059a3f2a3d9befa20f73d0ed43633
Normal-301.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-301.png,Normal/images/Normal-301.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-301.png,Normal/masks/Normal-301.png,True,False,299,299,L,136.35,4f9a22cfae769395bca14f56a8aeed5a
Normal-3010.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3010.png,Normal/images/Normal-3010.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3010.png,Normal/masks/Normal-3010.png,True,False,299,299,L,160.59,fa12e8150e32e98c58f6c5c35d7b1581
Normal-3011.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3011.png,Normal/images/Normal-3011.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3011.png,Normal/masks/Normal-3011.png,True,False,299,299,L,99.64,7626a064730e6ebae16c4f903e5763f1
Normal-3012.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3012.png,Normal/images/Normal-3012.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3012.png,Normal/masks/Normal-3012.png,True,False,299,299,L,125.1,8f811b52ce1a150842397d2cf62a9ce1
Normal-3013.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3013.png,Normal/images/Normal-3013.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3013.png,Normal/masks/Normal-3013.png,True,False,299,299,L,102.1,3fb1358bd76c16a40c5d7989bffca24a
Normal-3014.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3014.png,Normal/images/Normal-3014.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3014.png,Normal/masks/Normal-3014.png,True,False,299,299,L,129.28,f740b86273a8d69d87469e2c995609fd
Normal-3015.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3015.png,Normal/images/Normal-3015.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3015.png,Normal/masks/Normal-3015.png,True,False,299,299,L,107.39,2f37576407ef001087fadffc0bf479b3
Normal-3016.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3016.png,Normal/images/Normal-3016.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3016.png,Normal/masks/Normal-3016.png,True,False,299,299,L,115.97,fc5c9c9cb3252b50320a01e8a6fdb722
Normal-3017.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3017.png,Normal/images/Normal-3017.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3017.png,Normal/masks/Normal-3017.png,True,False,299,299,L,171.1,52c5ceef1b62364e91d33aca3e7af250
Normal-3018.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3018.png,Normal/images/Normal-3018.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3018.png,Normal/masks/Normal-3018.png,True,False,299,299,L,125.32,fbcbdf8dd47583dc1ec48c4d068f7c50
Normal-3019.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3019.png,Normal/images/Normal-3019.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3019.png,Normal/masks/Normal-3019.png,True,False,299,299,L,178.22,ee6be4c1bed13fa7040b3db9ef183e42
Normal-302.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-302.png,Normal/images/Normal-302.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-302.png,Normal/masks/Normal-302.png,True,False,299,299,L,128.92,c315d029110306d52cd4d906c6965d79
Normal-3020.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3020.png,Normal/images/Normal-3020.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3020.png,Normal/masks/Normal-3020.png,True,False,299,299,L,124.76,dbbcfcb1807cdb7667f791745ef93f46
Normal-3021.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3021.png,Normal/images/Normal-3021.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3021.png,Normal/masks/Normal-3021.png,True,False,299,299,L,154.42,c220bb042507620afb9b5e76f2a2b440
Normal-3022.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3022.png,Normal/images/Normal-3022.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3022.png,Normal/masks/Normal-3022.png,True,False,299,299,L,96.45,f880829f5037b7c6d0355e2d303411a3
Normal-3023.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3023.png,Normal/images/Normal-3023.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3023.png,Normal/masks/Normal-3023.png,True,False,299,299,L,148.44,e2586dc11b7a7cf93d97b2b94dbffa11
Normal-3024.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3024.png,Normal/images/Normal-3024.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3024.png,Normal/masks/Normal-3024.png,True,False,299,299,L,123.82,dd9b24e73f3ed684731287d329f7f329
Normal-3025.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3025.png,Normal/images/Normal-3025.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3025.png,Normal/masks/Normal-3025.png,True,False,299,299,L,148.74,23bcc49329dae4d2f3c4d5c54ceb9418
Normal-3026.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3026.png,Normal/images/Normal-3026.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3026.png,Normal/masks/Normal-3026.png,True,False,299,299,L,129.32,21715ecb8974f8d664732f31467c79fe
Normal-3027.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3027.png,Normal/images/Normal-3027.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3027.png,Normal/masks/Normal-3027.png,True,False,299,299,L,90.73,4cb119ce754ac1ac5bc7910805d79f94
Normal-3028.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3028.png,Normal/images/Normal-3028.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3028.png,Normal/masks/Normal-3028.png,True,False,299,299,L,176.13,651f04be2c79e49472056b7e298497e2
Normal-3029.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3029.png,Normal/images/Normal-3029.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3029.png,Normal/masks/Normal-3029.png,True,False,299,299,L,103.35,5fa9e90a217a06258427064f2716c5ad
Normal-303.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-303.png,Normal/images/Normal-303.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-303.png,Normal/masks/Normal-303.png,True,False,299,299,L,117.71,e2db029f7fe307d3e3507cc6e1d42e86
Normal-3030.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3030.png,Normal/images/Normal-3030.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3030.png,Normal/masks/Normal-3030.png,True,False,299,299,L,122.32,e7a543cca900fa0bb2cb3e1e4962ef4f
Normal-3031.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3031.png,Normal/images/Normal-3031.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3031.png,Normal/masks/Normal-3031.png,True,False,299,299,L,117.74,d8ff2202264f56d06157281725dd0414
Normal-3032.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3032.png,Normal/images/Normal-3032.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3032.png,Normal/masks/Normal-3032.png,True,False,299,299,L,104.82,9f463facbb88e6be3249ea7ae2092cb0
Normal-3033.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3033.png,Normal/images/Normal-3033.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3033.png,Normal/masks/Normal-3033.png,True,False,299,299,L,158.9,9103ad84600546bd2a8f390903b934cf
Normal-3034.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3034.png,Normal/images/Normal-3034.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3034.png,Normal/masks/Normal-3034.png,True,False,299,299,L,110.47,c26480c04185977a791301f7153a8bf1
Normal-3035.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3035.png,Normal/images/Normal-3035.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3035.png,Normal/masks/Normal-3035.png,True,False,299,299,L,118.92,d338795dee39eb0e68e9cd2cf19c09ff
Normal-3036.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3036.png,Normal/images/Normal-3036.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3036.png,Normal/masks/Normal-3036.png,True,False,299,299,L,126.82,1c2a141c2b39fd8def02da35e91534fd
Normal-3037.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3037.png,Normal/images/Normal-3037.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3037.png,Normal/masks/Normal-3037.png,True,False,299,299,L,105.67,94abbc46b3cb0d66656101ac758aba1a
Normal-3038.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3038.png,Normal/images/Normal-3038.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3038.png,Normal/masks/Normal-3038.png,True,False,299,299,L,148.75,3aab7fba78b174566fa867f33c00c9a6
Normal-3039.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3039.png,Normal/images/Normal-3039.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3039.png,Normal/masks/Normal-3039.png,True,False,299,299,L,117.04,04002cb09a910ddaa9cc4921936dc976
Normal-304.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-304.png,Normal/images/Normal-304.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-304.png,Normal/masks/Normal-304.png,True,False,299,299,L,120.39,d50ea8da9e4ec98f8ecc4b67c688f298
Normal-3040.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3040.png,Normal/images/Normal-3040.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3040.png,Normal/masks/Normal-3040.png,True,False,299,299,L,114.42,498b1e3e643850c70cef15fac701cfb6
Normal-3041.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3041.png,Normal/images/Normal-3041.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3041.png,Normal/masks/Normal-3041.png,True,False,299,299,L,156.47,876b42016b3305a6342801c5f12d2eff
Normal-3042.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3042.png,Normal/images/Normal-3042.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3042.png,Normal/masks/Normal-3042.png,True,False,299,299,L,172.52,7e8fe150388d5ea76d768feea2bd78f8
Normal-3043.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3043.png,Normal/images/Normal-3043.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3043.png,Normal/masks/Normal-3043.png,True,False,299,299,L,164.43,1fd8e047c0eb829c05a4adc2a20d3042
Normal-3044.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3044.png,Normal/images/Normal-3044.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3044.png,Normal/masks/Normal-3044.png,True,False,299,299,L,106.89,24af39aeede581aa8d19fae9ac2c920f
Normal-3045.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3045.png,Normal/images/Normal-3045.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3045.png,Normal/masks/Normal-3045.png,True,False,299,299,L,110.26,fcce05dc6da0b6c8293d0ff69bbeae02
Normal-3046.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3046.png,Normal/images/Normal-3046.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3046.png,Normal/masks/Normal-3046.png,True,False,299,299,L,186.75,9c8bbd747f1add94bb114bd5506f5647
Normal-3047.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3047.png,Normal/images/Normal-3047.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3047.png,Normal/masks/Normal-3047.png,True,False,299,299,L,157.39,d30e18be4443f3fd82000f4241d454dd
Normal-3048.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3048.png,Normal/images/Normal-3048.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3048.png,Normal/masks/Normal-3048.png,True,False,299,299,L,149.34,d6b0862acbe5fbda069716ae2e7af367
Normal-3049.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3049.png,Normal/images/Normal-3049.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3049.png,Normal/masks/Normal-3049.png,True,False,299,299,L,109.79,8b30bda50fffdbc5474dae57fdfe94b1
Normal-305.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-305.png,Normal/images/Normal-305.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-305.png,Normal/masks/Normal-305.png,True,False,299,299,L,110.41,af53600f0f663daa47d39c18ce3e8f9e
Normal-3050.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3050.png,Normal/images/Normal-3050.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3050.png,Normal/masks/Normal-3050.png,True,False,299,299,L,98.38,11085ce90b6f9cdd7e38e51212183364
Normal-3051.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3051.png,Normal/images/Normal-3051.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3051.png,Normal/masks/Normal-3051.png,True,False,299,299,L,169.9,81e971d46ab142f0c96dfca800a42fc3
Normal-3052.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3052.png,Normal/images/Normal-3052.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3052.png,Normal/masks/Normal-3052.png,True,False,299,299,L,122.17,26f14fbe97413ef241f36f90f63b1a92
Normal-3053.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3053.png,Normal/images/Normal-3053.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3053.png,Normal/masks/Normal-3053.png,True,False,299,299,L,142.08,8fcd5b2c0e11b02e542998157944e829
Normal-3054.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3054.png,Normal/images/Normal-3054.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3054.png,Normal/masks/Normal-3054.png,True,False,299,299,L,124.36,508dc1d9f0491c4839a0c164a843717f
Normal-3055.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3055.png,Normal/images/Normal-3055.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3055.png,Normal/masks/Normal-3055.png,True,False,299,299,L,117.19,8580d0e5abfcbd172bc7eb3ad09ab9e1
Normal-3056.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3056.png,Normal/images/Normal-3056.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3056.png,Normal/masks/Normal-3056.png,True,False,299,299,L,147.84,201b284ecc9e3d32d886b1873ef2167c
Normal-3057.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3057.png,Normal/images/Normal-3057.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3057.png,Normal/masks/Normal-3057.png,True,False,299,299,L,118.23,0b09a6341d720d86218dd3fcd3fff242
Normal-3058.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3058.png,Normal/images/Normal-3058.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3058.png,Normal/masks/Normal-3058.png,True,False,299,299,L,165.4,e27ec2614d9d75da7a5142459fdc0c0d
Normal-3059.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3059.png,Normal/images/Normal-3059.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3059.png,Normal/masks/Normal-3059.png,True,False,299,299,L,168.47,866329797840f03924e0fd6ad93cf7ee
Normal-306.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-306.png,Normal/images/Normal-306.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-306.png,Normal/masks/Normal-306.png,True,False,299,299,L,135.75,3b2db87344479b33c8fdec1c2750dbf1
Normal-3060.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3060.png,Normal/images/Normal-3060.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3060.png,Normal/masks/Normal-3060.png,True,False,299,299,L,159.43,7dbc88b221d8dc4403fbffcf11db0ccc
Normal-3061.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3061.png,Normal/images/Normal-3061.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3061.png,Normal/masks/Normal-3061.png,True,False,299,299,L,118.2,1af0374da3aa5a85ee08501a5fd54fc6
Normal-3062.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3062.png,Normal/images/Normal-3062.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3062.png,Normal/masks/Normal-3062.png,True,False,299,299,L,188.36,4cf8fd15ee7bd85d9d09c8b14f179f1d
Normal-3063.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3063.png,Normal/images/Normal-3063.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3063.png,Normal/masks/Normal-3063.png,True,False,299,299,L,104.79,3c621f09055deaa4084bbf3ec11f1000
Normal-3064.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3064.png,Normal/images/Normal-3064.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3064.png,Normal/masks/Normal-3064.png,True,False,299,299,L,128.91,ee2db802dad915dfa39693f716b4c0c6
Normal-3065.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3065.png,Normal/images/Normal-3065.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3065.png,Normal/masks/Normal-3065.png,True,False,299,299,L,139.75,8abe6409a95bdd157a17ce9d5e4873a8
Normal-3066.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3066.png,Normal/images/Normal-3066.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3066.png,Normal/masks/Normal-3066.png,True,False,299,299,L,143.84,5cb1a2424dc144b8658cdf00d36aeaa8
Normal-3067.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3067.png,Normal/images/Normal-3067.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3067.png,Normal/masks/Normal-3067.png,True,False,299,299,L,159.76,aebc7a47844abcd035fc7607749541f9
Normal-3068.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3068.png,Normal/images/Normal-3068.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3068.png,Normal/masks/Normal-3068.png,True,False,299,299,L,127.82,073927810f26442787d6aafe978bab4c
Normal-3069.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3069.png,Normal/images/Normal-3069.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3069.png,Normal/masks/Normal-3069.png,True,False,299,299,L,110.79,eac68a5a79a0347692970f553b38f413
Normal-307.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-307.png,Normal/images/Normal-307.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-307.png,Normal/masks/Normal-307.png,True,False,299,299,L,110.77,29fe44489d8ed48fc9eaad4e8a2cba72
Normal-3070.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3070.png,Normal/images/Normal-3070.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3070.png,Normal/masks/Normal-3070.png,True,False,299,299,L,141.64,f559fd5c5af6e958ff41f928f23ea7fd
Normal-3071.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3071.png,Normal/images/Normal-3071.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3071.png,Normal/masks/Normal-3071.png,True,False,299,299,L,171.42,93aaed0e3107c561b75fe7662111027f
Normal-3072.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3072.png,Normal/images/Normal-3072.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3072.png,Normal/masks/Normal-3072.png,True,False,299,299,L,118.32,7a0213b4d00f25d4fc9d7c9e07bf2fdf
Normal-3073.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3073.png,Normal/images/Normal-3073.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3073.png,Normal/masks/Normal-3073.png,True,False,299,299,L,104.73,515a266a3225e582912e04fde0764595
Normal-3074.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3074.png,Normal/images/Normal-3074.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3074.png,Normal/masks/Normal-3074.png,True,False,299,299,L,117.78,b28e0fc4f9d346252ae6d28962b0fdb0
Normal-3075.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3075.png,Normal/images/Normal-3075.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3075.png,Normal/masks/Normal-3075.png,True,False,299,299,L,117.48,0e638a4652d528e1930cffa269f3d92d
Normal-3076.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3076.png,Normal/images/Normal-3076.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3076.png,Normal/masks/Normal-3076.png,True,False,299,299,L,112.07,e2eae4641937c3fee9d6cc0a21c5efcc
Normal-3077.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3077.png,Normal/images/Normal-3077.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3077.png,Normal/masks/Normal-3077.png,True,False,299,299,L,104.47,6057c939b6878470a93039dc8a9461e9
Normal-3078.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3078.png,Normal/images/Normal-3078.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3078.png,Normal/masks/Normal-3078.png,True,False,299,299,L,109.32,5a15ccd36b472014b686191db71cfb3f
Normal-3079.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3079.png,Normal/images/Normal-3079.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3079.png,Normal/masks/Normal-3079.png,True,False,299,299,L,133.68,dc24c88366125df378b048da80897b4d
Normal-308.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-308.png,Normal/images/Normal-308.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-308.png,Normal/masks/Normal-308.png,True,False,299,299,L,125.86,39e89790dad854adef50d5283fb2b9e0
Normal-3080.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3080.png,Normal/images/Normal-3080.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3080.png,Normal/masks/Normal-3080.png,True,False,299,299,L,126.68,ac14d4bfde462eaa22521410e3a2b7d4
Normal-3081.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3081.png,Normal/images/Normal-3081.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3081.png,Normal/masks/Normal-3081.png,True,False,299,299,L,115.88,c78ea5e199e1af35833c0fed9b3d766c
Normal-3082.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3082.png,Normal/images/Normal-3082.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3082.png,Normal/masks/Normal-3082.png,True,False,299,299,L,115.71,9810309204105d86b257bea810332358
Normal-3083.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3083.png,Normal/images/Normal-3083.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3083.png,Normal/masks/Normal-3083.png,True,False,299,299,L,146.75,b462b2f0e0caca2020adf80773cf87aa
Normal-3084.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3084.png,Normal/images/Normal-3084.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3084.png,Normal/masks/Normal-3084.png,True,False,299,299,L,87.44,f8aa92914b6a74188e3fafd3223a746d
Normal-3085.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3085.png,Normal/images/Normal-3085.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3085.png,Normal/masks/Normal-3085.png,True,False,299,299,L,117.05,d6aa2ba0e4fd5fa14ff1f20bd55d09ae
Normal-3086.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3086.png,Normal/images/Normal-3086.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3086.png,Normal/masks/Normal-3086.png,True,False,299,299,L,100.54,357e4c805f6fea445621d87a99c7675e
Normal-3087.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3087.png,Normal/images/Normal-3087.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3087.png,Normal/masks/Normal-3087.png,True,False,299,299,L,132.06,149c359ef327716e0679d9e7e37b41d2
Normal-3088.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3088.png,Normal/images/Normal-3088.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3088.png,Normal/masks/Normal-3088.png,True,False,299,299,L,96.07,22b53f78ee2374722f3acae09af4d5e0
Normal-3089.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3089.png,Normal/images/Normal-3089.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3089.png,Normal/masks/Normal-3089.png,True,False,299,299,L,150.51,9a48ed49ca911d175b8f31b00f5f2cd1
Normal-309.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-309.png,Normal/images/Normal-309.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-309.png,Normal/masks/Normal-309.png,True,False,299,299,L,126.09,1eb534560570d5c44a19e724e74aea13
Normal-3090.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3090.png,Normal/images/Normal-3090.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3090.png,Normal/masks/Normal-3090.png,True,False,299,299,L,153.38,6c1dde331fd7fabfb6e3174f818f9167
Normal-3091.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3091.png,Normal/images/Normal-3091.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3091.png,Normal/masks/Normal-3091.png,True,False,299,299,L,167.13,4e7dd26ce09c58d88a850edb56432500
Normal-3092.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3092.png,Normal/images/Normal-3092.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3092.png,Normal/masks/Normal-3092.png,True,False,299,299,L,106.43,2bf7a893f662e4d68852f0c064cc6ebf
Normal-3093.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3093.png,Normal/images/Normal-3093.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3093.png,Normal/masks/Normal-3093.png,True,False,299,299,L,130.54,a970e4cec4048821970536e91c2586d0
Normal-3094.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3094.png,Normal/images/Normal-3094.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3094.png,Normal/masks/Normal-3094.png,True,False,299,299,L,116.12,d46a19691d8e111d538089ea616f39a6
Normal-3095.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3095.png,Normal/images/Normal-3095.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3095.png,Normal/masks/Normal-3095.png,True,False,299,299,L,128.81,337f8017ca787eb3ba451fb0a71253bd
Normal-3096.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3096.png,Normal/images/Normal-3096.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3096.png,Normal/masks/Normal-3096.png,True,False,299,299,L,132.0,c5982fdafb66e88e1fc6232a3674c3e1
Normal-3097.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3097.png,Normal/images/Normal-3097.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3097.png,Normal/masks/Normal-3097.png,True,False,299,299,L,123.03,7c6b64b78775e54154018bdfdeb8530e
Normal-3098.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3098.png,Normal/images/Normal-3098.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3098.png,Normal/masks/Normal-3098.png,True,False,299,299,L,119.36,11c59f8e5c163ea6f50fee34d066e3a8
Normal-3099.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3099.png,Normal/images/Normal-3099.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3099.png,Normal/masks/Normal-3099.png,True,False,299,299,L,122.81,714a977cbefdf3b10b0b17de70757b4f
Normal-31.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-31.png,Normal/images/Normal-31.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-31.png,Normal/masks/Normal-31.png,True,False,299,299,L,108.28,24251ae0f604f069e10265f46c6425ca
Normal-310.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-310.png,Normal/images/Normal-310.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-310.png,Normal/masks/Normal-310.png,True,False,299,299,L,116.79,49c4b349126155ba43de195b16085309
Normal-3100.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3100.png,Normal/images/Normal-3100.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3100.png,Normal/masks/Normal-3100.png,True,False,299,299,L,132.75,25d269c46feabcee53a3b772750490fe
Normal-3101.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3101.png,Normal/images/Normal-3101.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3101.png,Normal/masks/Normal-3101.png,True,False,299,299,L,104.14,ebd0775d81d17337b1fe8af212f013d1
Normal-3102.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3102.png,Normal/images/Normal-3102.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3102.png,Normal/masks/Normal-3102.png,True,False,299,299,L,126.25,63e29d0735fa9ee0014fde23e342f761
Normal-3103.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3103.png,Normal/images/Normal-3103.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3103.png,Normal/masks/Normal-3103.png,True,False,299,299,L,175.67,e750b20356097556f5ab52bb3046d3ab
Normal-3104.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3104.png,Normal/images/Normal-3104.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3104.png,Normal/masks/Normal-3104.png,True,False,299,299,L,118.09,bcc6d97184d5452f86457082550a3c85
Normal-3105.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3105.png,Normal/images/Normal-3105.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3105.png,Normal/masks/Normal-3105.png,True,False,299,299,L,107.34,586ca87fda5d1ee0a25369239ba2d5c9
Normal-3106.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3106.png,Normal/images/Normal-3106.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3106.png,Normal/masks/Normal-3106.png,True,False,299,299,L,132.34,f5c3633c1f69c184636b7057f4ba3924
Normal-3107.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3107.png,Normal/images/Normal-3107.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3107.png,Normal/masks/Normal-3107.png,True,False,299,299,L,105.04,519a15dd911c5fc20829b3f360c9735b
Normal-3108.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3108.png,Normal/images/Normal-3108.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3108.png,Normal/masks/Normal-3108.png,True,False,299,299,L,142.18,3adcab7a46bce2f20075226d27f2ba60
Normal-3109.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3109.png,Normal/images/Normal-3109.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3109.png,Normal/masks/Normal-3109.png,True,False,299,299,L,124.03,dafe80d7369086912a393201c49008a8
Normal-311.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-311.png,Normal/images/Normal-311.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-311.png,Normal/masks/Normal-311.png,True,False,299,299,L,98.53,c29617b92116af444365f9e8c7710d5b
Normal-3110.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3110.png,Normal/images/Normal-3110.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3110.png,Normal/masks/Normal-3110.png,True,False,299,299,L,121.19,6afa3ef675512b8bdc42389245b148f8
Normal-3111.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3111.png,Normal/images/Normal-3111.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3111.png,Normal/masks/Normal-3111.png,True,False,299,299,L,122.93,f49939ab8ceac624fc656c0bc7a03c2c
Normal-3112.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3112.png,Normal/images/Normal-3112.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3112.png,Normal/masks/Normal-3112.png,True,False,299,299,L,114.44,bd9cd48c751785615db2d67d8fc8d9e2
Normal-3113.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3113.png,Normal/images/Normal-3113.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3113.png,Normal/masks/Normal-3113.png,True,False,299,299,L,120.27,1b093447e28d9ed92d88df636c5c1be3
Normal-3114.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3114.png,Normal/images/Normal-3114.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3114.png,Normal/masks/Normal-3114.png,True,False,299,299,L,136.08,bc0a3bb6a6d586851a7319a41d447877
Normal-3115.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3115.png,Normal/images/Normal-3115.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3115.png,Normal/masks/Normal-3115.png,True,False,299,299,L,123.31,b21fb744fc1ac22a56f1d2850b19b7c7
Normal-3116.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3116.png,Normal/images/Normal-3116.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3116.png,Normal/masks/Normal-3116.png,True,False,299,299,L,148.16,d81ebfaf30457ae87040bdc8641beadc
Normal-3117.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3117.png,Normal/images/Normal-3117.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3117.png,Normal/masks/Normal-3117.png,True,False,299,299,L,114.54,8deb1bd9671d12d2225f4902bc0c9b17
Normal-3118.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3118.png,Normal/images/Normal-3118.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3118.png,Normal/masks/Normal-3118.png,True,False,299,299,L,121.57,63e585db86f996f269948f1e70d5b766
Normal-3119.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3119.png,Normal/images/Normal-3119.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3119.png,Normal/masks/Normal-3119.png,True,False,299,299,L,115.56,6ae0cea582d050d8096d1400aa53a43b
Normal-312.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-312.png,Normal/images/Normal-312.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-312.png,Normal/masks/Normal-312.png,True,False,299,299,L,129.12,f2edd99f7f48625d4311cc6513227bc4
Normal-3120.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3120.png,Normal/images/Normal-3120.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3120.png,Normal/masks/Normal-3120.png,True,False,299,299,L,181.11,37098ca812d0cd018ddae24ea3dd43a6
Normal-3121.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3121.png,Normal/images/Normal-3121.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3121.png,Normal/masks/Normal-3121.png,True,False,299,299,L,119.37,17039f9521fe6df6732e63d4068b839a
Normal-3122.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3122.png,Normal/images/Normal-3122.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3122.png,Normal/masks/Normal-3122.png,True,False,299,299,L,127.92,2c1c7f502f3fde43b32443f3d9b2f70c
Normal-3123.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3123.png,Normal/images/Normal-3123.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3123.png,Normal/masks/Normal-3123.png,True,False,299,299,L,107.81,dc73a59b2d98e5d8b5fce93c974b4571
Normal-3124.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3124.png,Normal/images/Normal-3124.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3124.png,Normal/masks/Normal-3124.png,True,False,299,299,L,113.32,1d77740c536eadd532cb2644866948fb
Normal-3125.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3125.png,Normal/images/Normal-3125.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3125.png,Normal/masks/Normal-3125.png,True,False,299,299,L,145.37,27f877065873a39f7d694ac8071e5416
Normal-3126.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3126.png,Normal/images/Normal-3126.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3126.png,Normal/masks/Normal-3126.png,True,False,299,299,L,121.11,9e0da15ef8dba52d230acfaeee32d665
Normal-3127.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3127.png,Normal/images/Normal-3127.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3127.png,Normal/masks/Normal-3127.png,True,False,299,299,L,121.89,d57ff3e79ab4f7773060f29b4af04457
Normal-3128.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3128.png,Normal/images/Normal-3128.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3128.png,Normal/masks/Normal-3128.png,True,False,299,299,L,111.88,b2ec8d1040f90181a3c08661135c5296
Normal-3129.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3129.png,Normal/images/Normal-3129.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3129.png,Normal/masks/Normal-3129.png,True,False,299,299,L,127.06,395d291f75f9e43d97f83406a6d6bc04
Normal-313.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-313.png,Normal/images/Normal-313.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-313.png,Normal/masks/Normal-313.png,True,False,299,299,L,118.19,cf4a0e504ade67e023786007e0633359
Normal-3130.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3130.png,Normal/images/Normal-3130.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3130.png,Normal/masks/Normal-3130.png,True,False,299,299,L,155.53,1f804c2ab1a39e93acafad3dfbf1d239
Normal-3131.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3131.png,Normal/images/Normal-3131.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3131.png,Normal/masks/Normal-3131.png,True,False,299,299,L,167.79,6aa6a457d2cfd79cb7147af08bebeea3
Normal-3132.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3132.png,Normal/images/Normal-3132.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3132.png,Normal/masks/Normal-3132.png,True,False,299,299,L,175.18,dfd758a162fbae1a913515190c44a464
Normal-3133.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3133.png,Normal/images/Normal-3133.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3133.png,Normal/masks/Normal-3133.png,True,False,299,299,L,174.55,c4e223e20ba05c88f18f6d465d00d0c8
Normal-3134.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3134.png,Normal/images/Normal-3134.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3134.png,Normal/masks/Normal-3134.png,True,False,299,299,L,98.44,06d0528b29fb39fbb9f85aa1af2d3195
Normal-3135.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3135.png,Normal/images/Normal-3135.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3135.png,Normal/masks/Normal-3135.png,True,False,299,299,L,174.85,5a53c7622aba6dba0a4664b4e3d76ec5
Normal-3136.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3136.png,Normal/images/Normal-3136.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3136.png,Normal/masks/Normal-3136.png,True,False,299,299,L,141.09,6acb06377c6e4e8b5fec33eba236bf05
Normal-3137.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3137.png,Normal/images/Normal-3137.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3137.png,Normal/masks/Normal-3137.png,True,False,299,299,L,171.37,f5172464077b7e3c04db15c60804db16
Normal-3138.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3138.png,Normal/images/Normal-3138.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3138.png,Normal/masks/Normal-3138.png,True,False,299,299,L,119.68,4c7d606745cc115f3f7ad0ee60df4837
Normal-3139.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3139.png,Normal/images/Normal-3139.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3139.png,Normal/masks/Normal-3139.png,True,False,299,299,L,109.71,7cc44a3b6ab9daa7c2de653d7db552f5
Normal-314.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-314.png,Normal/images/Normal-314.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-314.png,Normal/masks/Normal-314.png,True,False,299,299,L,113.4,9466dc7d49a68a22e17bc6b218329d82
Normal-3140.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3140.png,Normal/images/Normal-3140.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3140.png,Normal/masks/Normal-3140.png,True,False,299,299,L,123.22,f9c965ebb665f0b4fa809976a8c87929
Normal-3141.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3141.png,Normal/images/Normal-3141.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3141.png,Normal/masks/Normal-3141.png,True,False,299,299,L,175.82,f6a3b4f27f9a369d1cbd51be9542a5ed
Normal-3142.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3142.png,Normal/images/Normal-3142.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3142.png,Normal/masks/Normal-3142.png,True,False,299,299,L,139.12,0369dd7e4d9364934137d03b628383ba
Normal-3143.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3143.png,Normal/images/Normal-3143.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3143.png,Normal/masks/Normal-3143.png,True,False,299,299,L,126.0,0935d905258cd9887fb3e7693945454d
Normal-3144.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3144.png,Normal/images/Normal-3144.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3144.png,Normal/masks/Normal-3144.png,True,False,299,299,L,114.73,e21aa358e4f93c9f5f65eaad3ae17cc5
Normal-3145.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3145.png,Normal/images/Normal-3145.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3145.png,Normal/masks/Normal-3145.png,True,False,299,299,L,128.89,a3a8a7e4d5dc2ec8992acab94ca345e8
Normal-3146.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3146.png,Normal/images/Normal-3146.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3146.png,Normal/masks/Normal-3146.png,True,False,299,299,L,153.53,33854e0bb32fabe9f36eebdc65ff546b
Normal-3147.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3147.png,Normal/images/Normal-3147.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3147.png,Normal/masks/Normal-3147.png,True,False,299,299,L,121.63,0d81169f64a8f2ee8754161bc079c8e8
Normal-3148.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3148.png,Normal/images/Normal-3148.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3148.png,Normal/masks/Normal-3148.png,True,False,299,299,L,109.07,e6aef47399deca29afded4fb05672a08
Normal-3149.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3149.png,Normal/images/Normal-3149.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3149.png,Normal/masks/Normal-3149.png,True,False,299,299,L,93.67,6b5e9755e371581049553d9957e23311
Normal-315.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-315.png,Normal/images/Normal-315.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-315.png,Normal/masks/Normal-315.png,True,False,299,299,L,115.15,7d037177bda90c17d9b30dd1db58b48d
Normal-3150.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3150.png,Normal/images/Normal-3150.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3150.png,Normal/masks/Normal-3150.png,True,False,299,299,L,175.61,b3971201f012ec9e7b9843b4afc9c3c9
Normal-3151.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3151.png,Normal/images/Normal-3151.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3151.png,Normal/masks/Normal-3151.png,True,False,299,299,L,138.63,a79b94f870ae82b0b6049124ddad61e4
Normal-3152.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3152.png,Normal/images/Normal-3152.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3152.png,Normal/masks/Normal-3152.png,True,False,299,299,L,115.63,6979809eb49edf60c84ca8c04156e339
Normal-3153.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3153.png,Normal/images/Normal-3153.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3153.png,Normal/masks/Normal-3153.png,True,False,299,299,L,146.64,3a45bd8ba7c19a0fb641d70dd48042f2
Normal-3154.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3154.png,Normal/images/Normal-3154.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3154.png,Normal/masks/Normal-3154.png,True,False,299,299,L,159.62,dae324fb54cf536564ac3bc0af886b03
Normal-3155.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3155.png,Normal/images/Normal-3155.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3155.png,Normal/masks/Normal-3155.png,True,False,299,299,L,123.87,813f94a14e3c8ba3253d43c47e5e2543
Normal-3156.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3156.png,Normal/images/Normal-3156.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3156.png,Normal/masks/Normal-3156.png,True,False,299,299,L,159.75,6c05ca7e8d85a1e4c4d53c0fca8705d7
Normal-3157.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3157.png,Normal/images/Normal-3157.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3157.png,Normal/masks/Normal-3157.png,True,False,299,299,L,108.15,892cd52f53926f7f8d59cac5ef37010c
Normal-3158.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3158.png,Normal/images/Normal-3158.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3158.png,Normal/masks/Normal-3158.png,True,False,299,299,L,106.1,8384fe274133a25a1ce360804bf681b4
Normal-3159.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3159.png,Normal/images/Normal-3159.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3159.png,Normal/masks/Normal-3159.png,True,False,299,299,L,119.67,bb7724f738e70ca32af58e53cbf309c3
Normal-316.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-316.png,Normal/images/Normal-316.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-316.png,Normal/masks/Normal-316.png,True,False,299,299,L,90.31,71a46f489fe84c7e823cf82707ded17d
Normal-3160.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3160.png,Normal/images/Normal-3160.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3160.png,Normal/masks/Normal-3160.png,True,False,299,299,L,165.13,c4219467246e31fc8cc6928c3ea6380e
Normal-3161.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3161.png,Normal/images/Normal-3161.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3161.png,Normal/masks/Normal-3161.png,True,False,299,299,L,124.21,f076bb2e5895122e226e0640ee3674e4
Normal-3162.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3162.png,Normal/images/Normal-3162.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3162.png,Normal/masks/Normal-3162.png,True,False,299,299,L,134.28,7536072b4807877f7fd50d4e5a938884
Normal-3163.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3163.png,Normal/images/Normal-3163.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3163.png,Normal/masks/Normal-3163.png,True,False,299,299,L,132.9,0e1a3d669131576e5a46c98a4fe57fb2
Normal-3164.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3164.png,Normal/images/Normal-3164.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3164.png,Normal/masks/Normal-3164.png,True,False,299,299,L,101.4,aeba204de73dfcbdaa9fe8799b6e3c7b
Normal-3165.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3165.png,Normal/images/Normal-3165.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3165.png,Normal/masks/Normal-3165.png,True,False,299,299,L,165.04,b225cbe06d4ee1721b0310d719cc49b5
Normal-3166.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3166.png,Normal/images/Normal-3166.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3166.png,Normal/masks/Normal-3166.png,True,False,299,299,L,116.02,6b53c709365f77b6a1c8c5dd27de3480
Normal-3167.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3167.png,Normal/images/Normal-3167.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3167.png,Normal/masks/Normal-3167.png,True,False,299,299,L,167.98,601b305397907d5463b1fa459cfb767d
Normal-3168.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3168.png,Normal/images/Normal-3168.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3168.png,Normal/masks/Normal-3168.png,True,False,299,299,L,180.99,c877f99833661095e973e89651357309
Normal-3169.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3169.png,Normal/images/Normal-3169.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3169.png,Normal/masks/Normal-3169.png,True,False,299,299,L,130.64,dfa8c096d000719fa9fe9a3a6b953e91
Normal-317.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-317.png,Normal/images/Normal-317.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-317.png,Normal/masks/Normal-317.png,True,False,299,299,L,121.98,de4d558c6612d3a468df2c6dc96af016
Normal-3170.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3170.png,Normal/images/Normal-3170.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3170.png,Normal/masks/Normal-3170.png,True,False,299,299,L,148.83,726f46305831919252079647853fac6e
Normal-3171.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3171.png,Normal/images/Normal-3171.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3171.png,Normal/masks/Normal-3171.png,True,False,299,299,L,121.7,29d186a55ef16a362432564f9a365e3a
Normal-3172.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3172.png,Normal/images/Normal-3172.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3172.png,Normal/masks/Normal-3172.png,True,False,299,299,L,96.06,19a2428157d27a5cab29f2028d9e001b
Normal-3173.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3173.png,Normal/images/Normal-3173.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3173.png,Normal/masks/Normal-3173.png,True,False,299,299,L,97.75,784cf8a304946deccb592628e14b90db
Normal-3174.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3174.png,Normal/images/Normal-3174.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3174.png,Normal/masks/Normal-3174.png,True,False,299,299,L,106.12,5dee0dcdb1968bf4e0e82549a64d3db2
Normal-3175.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3175.png,Normal/images/Normal-3175.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3175.png,Normal/masks/Normal-3175.png,True,False,299,299,L,143.84,cab4d446ab0cebf98249b87b370a38e0
Normal-3176.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3176.png,Normal/images/Normal-3176.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3176.png,Normal/masks/Normal-3176.png,True,False,299,299,L,172.47,4bb2b9ba1b0509d6194bdc14806200cb
Normal-3177.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3177.png,Normal/images/Normal-3177.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3177.png,Normal/masks/Normal-3177.png,True,False,299,299,L,134.82,a4e90e749de0f5b61df02578522ddbcb
Normal-3178.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3178.png,Normal/images/Normal-3178.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3178.png,Normal/masks/Normal-3178.png,True,False,299,299,L,109.61,162b76d97ced4399602809ea7d0f9441
Normal-3179.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3179.png,Normal/images/Normal-3179.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3179.png,Normal/masks/Normal-3179.png,True,False,299,299,L,133.7,64ae733765748e12739281084377b741
Normal-318.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-318.png,Normal/images/Normal-318.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-318.png,Normal/masks/Normal-318.png,True,False,299,299,L,134.74,ed7f8dd06e13c0312c200b369474aaee
Normal-3180.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3180.png,Normal/images/Normal-3180.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3180.png,Normal/masks/Normal-3180.png,True,False,299,299,L,107.07,45c7e4b7ee54216ddf292b982c05209c
Normal-3181.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3181.png,Normal/images/Normal-3181.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3181.png,Normal/masks/Normal-3181.png,True,False,299,299,L,158.53,4dca92bb81d11543dea8e8152511a9b1
Normal-3182.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3182.png,Normal/images/Normal-3182.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3182.png,Normal/masks/Normal-3182.png,True,False,299,299,L,136.43,83853ea12f43886431a1832186e200fc
Normal-3183.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3183.png,Normal/images/Normal-3183.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3183.png,Normal/masks/Normal-3183.png,True,False,299,299,L,145.73,a74b555d5861f31065c42d3f4d0868b8
Normal-3184.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3184.png,Normal/images/Normal-3184.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3184.png,Normal/masks/Normal-3184.png,True,False,299,299,L,117.85,337acb7be9a3cabae0e90ae172e612cc
Normal-3185.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3185.png,Normal/images/Normal-3185.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3185.png,Normal/masks/Normal-3185.png,True,False,299,299,L,114.98,d3a86d08d68e98358ad25eb091954c84
Normal-3186.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3186.png,Normal/images/Normal-3186.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3186.png,Normal/masks/Normal-3186.png,True,False,299,299,L,118.62,af8bd4c6bf8c3e16bdc91e7f9740f5ee
Normal-3187.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3187.png,Normal/images/Normal-3187.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3187.png,Normal/masks/Normal-3187.png,True,False,299,299,L,110.64,63f4c86d9bbec95c1eb7e42e3c2ed9fb
Normal-3188.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3188.png,Normal/images/Normal-3188.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3188.png,Normal/masks/Normal-3188.png,True,False,299,299,L,144.26,3c0a44f8a1146e63acd8183da13578be
Normal-3189.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3189.png,Normal/images/Normal-3189.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3189.png,Normal/masks/Normal-3189.png,True,False,299,299,L,99.36,1099b7ff3322ff001d59b34bf07cf6b7
Normal-319.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-319.png,Normal/images/Normal-319.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-319.png,Normal/masks/Normal-319.png,True,False,299,299,L,113.8,c655a9ce187aa2e525bf769389f723d3
Normal-3190.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3190.png,Normal/images/Normal-3190.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3190.png,Normal/masks/Normal-3190.png,True,False,299,299,L,132.57,2ab9e12f118437fa2ac291d8cd4bde85
Normal-3191.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3191.png,Normal/images/Normal-3191.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3191.png,Normal/masks/Normal-3191.png,True,False,299,299,L,141.72,9fc929d6fe3a2e099bd5347ea33c807a
Normal-3192.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3192.png,Normal/images/Normal-3192.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3192.png,Normal/masks/Normal-3192.png,True,False,299,299,L,97.15,543f636bd86df4b3586228d8206e3aba
Normal-3193.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3193.png,Normal/images/Normal-3193.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3193.png,Normal/masks/Normal-3193.png,True,False,299,299,L,147.64,91b3a0017a521d7392a7dec6adcabec8
Normal-3194.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3194.png,Normal/images/Normal-3194.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3194.png,Normal/masks/Normal-3194.png,True,False,299,299,L,140.68,032fa522c84967834e8de640c8fb43f1
Normal-3195.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3195.png,Normal/images/Normal-3195.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3195.png,Normal/masks/Normal-3195.png,True,False,299,299,L,145.54,cb48d983df87f50547a53a34332c9dc9
Normal-3196.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3196.png,Normal/images/Normal-3196.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3196.png,Normal/masks/Normal-3196.png,True,False,299,299,L,174.43,674d8740bbc99129a7e7ab7b310b78d6
Normal-3197.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3197.png,Normal/images/Normal-3197.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3197.png,Normal/masks/Normal-3197.png,True,False,299,299,L,123.15,e386f4e2368140e6c4e4b829dc1570b5
Normal-3198.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3198.png,Normal/images/Normal-3198.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3198.png,Normal/masks/Normal-3198.png,True,False,299,299,L,127.25,ddc281acff6e9b6886d41b2849159590
Normal-3199.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3199.png,Normal/images/Normal-3199.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3199.png,Normal/masks/Normal-3199.png,True,False,299,299,L,109.94,8e074a91a4fa600267bb05f17d28028d
Normal-32.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-32.png,Normal/images/Normal-32.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-32.png,Normal/masks/Normal-32.png,True,False,299,299,L,121.87,28ff475313953f426c43ed36bc2550c6
Normal-320.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-320.png,Normal/images/Normal-320.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-320.png,Normal/masks/Normal-320.png,True,False,299,299,L,126.18,159eabf48d617b615c363de645a07c70
Normal-3200.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3200.png,Normal/images/Normal-3200.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3200.png,Normal/masks/Normal-3200.png,True,False,299,299,L,110.57,0cc91421dd91a24db296f319eba6d726
Normal-3201.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3201.png,Normal/images/Normal-3201.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3201.png,Normal/masks/Normal-3201.png,True,False,299,299,L,113.94,382a25f8e758849385096e7ff55993a5
Normal-3202.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3202.png,Normal/images/Normal-3202.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3202.png,Normal/masks/Normal-3202.png,True,False,299,299,L,106.84,10212a7369fae506cc2d0b6ffb9ec582
Normal-3203.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3203.png,Normal/images/Normal-3203.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3203.png,Normal/masks/Normal-3203.png,True,False,299,299,L,108.1,e05aedeb52493f3116e495adc738e046
Normal-3204.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3204.png,Normal/images/Normal-3204.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3204.png,Normal/masks/Normal-3204.png,True,False,299,299,L,107.88,a345730e82f7cc034516b3925e3d9fa9
Normal-3205.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3205.png,Normal/images/Normal-3205.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3205.png,Normal/masks/Normal-3205.png,True,False,299,299,L,152.89,d3655fbf57c369a24bff2a863d7a0c21
Normal-3206.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3206.png,Normal/images/Normal-3206.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3206.png,Normal/masks/Normal-3206.png,True,False,299,299,L,109.78,131076e4a4bb0274400cd219e9c2c736
Normal-3207.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3207.png,Normal/images/Normal-3207.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3207.png,Normal/masks/Normal-3207.png,True,False,299,299,L,163.61,3371bc842a29403e3d87f86be90cda0a
Normal-3208.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3208.png,Normal/images/Normal-3208.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3208.png,Normal/masks/Normal-3208.png,True,False,299,299,L,91.75,4d52ebddbb1d4d02c779918010a1f79a
Normal-3209.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3209.png,Normal/images/Normal-3209.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3209.png,Normal/masks/Normal-3209.png,True,False,299,299,L,130.99,ed7ca3687f1119e2d680d7ac3dbd51df
Normal-321.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-321.png,Normal/images/Normal-321.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-321.png,Normal/masks/Normal-321.png,True,False,299,299,L,123.57,5dfd69f799e3f9c3edbd993df71ec503
Normal-3210.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3210.png,Normal/images/Normal-3210.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3210.png,Normal/masks/Normal-3210.png,True,False,299,299,L,110.75,31ffe86b751919da8bcf51c536c3b96a
Normal-3211.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3211.png,Normal/images/Normal-3211.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3211.png,Normal/masks/Normal-3211.png,True,False,299,299,L,132.45,3c458a5bab66e5f6792f048950ecb07f
Normal-3212.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3212.png,Normal/images/Normal-3212.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3212.png,Normal/masks/Normal-3212.png,True,False,299,299,L,176.21,977ec85365024b83a45b33c8118ef901
Normal-3213.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3213.png,Normal/images/Normal-3213.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3213.png,Normal/masks/Normal-3213.png,True,False,299,299,L,116.26,a1cb6417f3270f78284cd4a14f516452
Normal-3214.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3214.png,Normal/images/Normal-3214.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3214.png,Normal/masks/Normal-3214.png,True,False,299,299,L,149.97,c743edeeb67cfd02ca5b567a19a2538f
Normal-3215.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3215.png,Normal/images/Normal-3215.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3215.png,Normal/masks/Normal-3215.png,True,False,299,299,L,156.15,587a39a28840c4a78c99670437db3cad
Normal-3216.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3216.png,Normal/images/Normal-3216.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3216.png,Normal/masks/Normal-3216.png,True,False,299,299,L,121.86,871d93e853cf7a0ec674eca19be1255b
Normal-3217.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3217.png,Normal/images/Normal-3217.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3217.png,Normal/masks/Normal-3217.png,True,False,299,299,L,150.95,604b877f1db7e66162475b81a3a02d6b
Normal-3218.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3218.png,Normal/images/Normal-3218.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3218.png,Normal/masks/Normal-3218.png,True,False,299,299,L,107.93,4e0dacd09b50d09dadedcb68008f2275
Normal-3219.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3219.png,Normal/images/Normal-3219.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3219.png,Normal/masks/Normal-3219.png,True,False,299,299,L,133.36,90952c93cd3a886487b014262bfeddcb
Normal-322.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-322.png,Normal/images/Normal-322.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-322.png,Normal/masks/Normal-322.png,True,False,299,299,L,136.14,782941d7356c66acbcaf7c7ecbfc1601
Normal-3220.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3220.png,Normal/images/Normal-3220.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3220.png,Normal/masks/Normal-3220.png,True,False,299,299,L,162.97,aeb979bbfbaf358781fec7df263ec124
Normal-3221.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3221.png,Normal/images/Normal-3221.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3221.png,Normal/masks/Normal-3221.png,True,False,299,299,L,126.09,8df649d269057f5c55c40e518196f822
Normal-3222.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3222.png,Normal/images/Normal-3222.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3222.png,Normal/masks/Normal-3222.png,True,False,299,299,L,114.6,fdffe2482cef0e0b689cfb39a4ae30db
Normal-3223.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3223.png,Normal/images/Normal-3223.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3223.png,Normal/masks/Normal-3223.png,True,False,299,299,L,111.96,a9f3cb4dee0109ac450e205054eb19c8
Normal-3224.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3224.png,Normal/images/Normal-3224.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3224.png,Normal/masks/Normal-3224.png,True,False,299,299,L,115.6,294458e7d5cd3be9a55f8e8986d32749
Normal-3225.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3225.png,Normal/images/Normal-3225.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3225.png,Normal/masks/Normal-3225.png,True,False,299,299,L,117.94,59bce67418fd8a4cb028f2a9bac98c66
Normal-3226.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3226.png,Normal/images/Normal-3226.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3226.png,Normal/masks/Normal-3226.png,True,False,299,299,L,155.34,fc3d5a972157b46967cad06bd6758477
Normal-3227.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3227.png,Normal/images/Normal-3227.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3227.png,Normal/masks/Normal-3227.png,True,False,299,299,L,132.58,01f28da1e78db722f5d1ee8c87f9bc68
Normal-3228.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3228.png,Normal/images/Normal-3228.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3228.png,Normal/masks/Normal-3228.png,True,False,299,299,L,117.16,cec40113092971552b1aa8910ee86b71
Normal-3229.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3229.png,Normal/images/Normal-3229.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3229.png,Normal/masks/Normal-3229.png,True,False,299,299,L,97.4,4f6375aff804ed5afdd55797af620348
Normal-323.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-323.png,Normal/images/Normal-323.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-323.png,Normal/masks/Normal-323.png,True,False,299,299,L,123.76,b480236d467d3b4887fac1dd03cfb1fb
Normal-3230.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3230.png,Normal/images/Normal-3230.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3230.png,Normal/masks/Normal-3230.png,True,False,299,299,L,98.31,973504d0baae4d44751df4e81075c9f5
Normal-3231.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3231.png,Normal/images/Normal-3231.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3231.png,Normal/masks/Normal-3231.png,True,False,299,299,L,102.3,1f21924165dcba83f9b1ec7783329f6c
Normal-3232.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3232.png,Normal/images/Normal-3232.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3232.png,Normal/masks/Normal-3232.png,True,False,299,299,L,128.53,9ea4de25976134a8663c4ba9f6aaa791
Normal-3233.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3233.png,Normal/images/Normal-3233.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3233.png,Normal/masks/Normal-3233.png,True,False,299,299,L,115.1,82079461dee3996451b8570246dce0f9
Normal-3234.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3234.png,Normal/images/Normal-3234.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3234.png,Normal/masks/Normal-3234.png,True,False,299,299,L,137.66,58f49f0432283b181153fed4a440413b
Normal-3235.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3235.png,Normal/images/Normal-3235.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3235.png,Normal/masks/Normal-3235.png,True,False,299,299,L,161.89,0bc5bacfe05135a6203e2a8cb76a008f
Normal-3236.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3236.png,Normal/images/Normal-3236.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3236.png,Normal/masks/Normal-3236.png,True,False,299,299,L,95.25,430c95656fbd1e4656475c770c6bbd13
Normal-3237.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3237.png,Normal/images/Normal-3237.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3237.png,Normal/masks/Normal-3237.png,True,False,299,299,L,157.2,8e3d16da267e8adf0387eb5ee22ea4a6
Normal-3238.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3238.png,Normal/images/Normal-3238.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3238.png,Normal/masks/Normal-3238.png,True,False,299,299,L,161.69,c0ab031df10a88703051229ce3b4b2de
Normal-3239.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3239.png,Normal/images/Normal-3239.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3239.png,Normal/masks/Normal-3239.png,True,False,299,299,L,115.24,c411211959f85c9627557a90eaedfdf5
Normal-324.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-324.png,Normal/images/Normal-324.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-324.png,Normal/masks/Normal-324.png,True,False,299,299,L,118.56,35e32ba7ffe45704eb2470262cadff7e
Normal-3240.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3240.png,Normal/images/Normal-3240.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3240.png,Normal/masks/Normal-3240.png,True,False,299,299,L,116.64,5d2c7e1fb6938358988d595afb9d0a72
Normal-3241.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3241.png,Normal/images/Normal-3241.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3241.png,Normal/masks/Normal-3241.png,True,False,299,299,L,147.73,f6f44545c4d49385554c12f50c201017
Normal-3242.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3242.png,Normal/images/Normal-3242.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3242.png,Normal/masks/Normal-3242.png,True,False,299,299,L,179.74,537a47e7e05d5b8e55e02fa4ffe9971f
Normal-3243.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3243.png,Normal/images/Normal-3243.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3243.png,Normal/masks/Normal-3243.png,True,False,299,299,L,133.88,5d5a0ec848b56a0da29eec2e2b15cb02
Normal-3244.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3244.png,Normal/images/Normal-3244.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3244.png,Normal/masks/Normal-3244.png,True,False,299,299,L,127.2,df62f8e8ac6041c861d4279d203b02e6
Normal-3245.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3245.png,Normal/images/Normal-3245.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3245.png,Normal/masks/Normal-3245.png,True,False,299,299,L,162.45,92f3df951ebc00734e99b4e4bc4ae8ea
Normal-3246.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3246.png,Normal/images/Normal-3246.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3246.png,Normal/masks/Normal-3246.png,True,False,299,299,L,159.41,eb88b1564be5827d76b9d00e28c290c8
Normal-3247.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3247.png,Normal/images/Normal-3247.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3247.png,Normal/masks/Normal-3247.png,True,False,299,299,L,109.96,7ee2cc7e3d1f3ef513e6077dee59e42e
Normal-3248.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3248.png,Normal/images/Normal-3248.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3248.png,Normal/masks/Normal-3248.png,True,False,299,299,L,162.03,c333f6dfd009563eddb37eb33dd6b9fb
Normal-3249.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3249.png,Normal/images/Normal-3249.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3249.png,Normal/masks/Normal-3249.png,True,False,299,299,L,118.21,73691e9d4d0d431e8387c0b3a80f9861
Normal-325.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-325.png,Normal/images/Normal-325.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-325.png,Normal/masks/Normal-325.png,True,False,299,299,L,126.66,fa1e4a449d25c9e4bc9a2f2bc0283288
Normal-3250.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3250.png,Normal/images/Normal-3250.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3250.png,Normal/masks/Normal-3250.png,True,False,299,299,L,169.71,5279a6d21769cb60ebf1e69c91fa8d45
Normal-3251.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3251.png,Normal/images/Normal-3251.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3251.png,Normal/masks/Normal-3251.png,True,False,299,299,L,158.03,69efdf4fbf2a239c1506c041340423d9
Normal-3252.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3252.png,Normal/images/Normal-3252.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3252.png,Normal/masks/Normal-3252.png,True,False,299,299,L,109.09,763e9a4babdfa4966ac712a32e84952b
Normal-3253.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3253.png,Normal/images/Normal-3253.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3253.png,Normal/masks/Normal-3253.png,True,False,299,299,L,147.15,aa7bd02a04f0d41cb9dbcf144899c7ce
Normal-3254.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3254.png,Normal/images/Normal-3254.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3254.png,Normal/masks/Normal-3254.png,True,False,299,299,L,114.82,468d691a3693462d4bfbd1473624e657
Normal-3255.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3255.png,Normal/images/Normal-3255.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3255.png,Normal/masks/Normal-3255.png,True,False,299,299,L,142.02,8b5516cf57683e07533fd1353eb55896
Normal-3256.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3256.png,Normal/images/Normal-3256.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3256.png,Normal/masks/Normal-3256.png,True,False,299,299,L,127.05,6603a1eef5adb854ec5bebaa65b432d5
Normal-3257.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3257.png,Normal/images/Normal-3257.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3257.png,Normal/masks/Normal-3257.png,True,False,299,299,L,134.91,bd8d521bf82ad3ecf19314653a8c1856
Normal-3258.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3258.png,Normal/images/Normal-3258.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3258.png,Normal/masks/Normal-3258.png,True,False,299,299,L,186.32,0b971b613f074e21f474e3d0d8c306c8
Normal-3259.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3259.png,Normal/images/Normal-3259.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3259.png,Normal/masks/Normal-3259.png,True,False,299,299,L,118.6,7326a9a91b36e7d59161c7ca493f8383
Normal-326.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-326.png,Normal/images/Normal-326.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-326.png,Normal/masks/Normal-326.png,True,False,299,299,L,111.31,82a0f1dd36f8ea2d1d3f652ce5a900f6
Normal-3260.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3260.png,Normal/images/Normal-3260.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3260.png,Normal/masks/Normal-3260.png,True,False,299,299,L,112.02,1645081754fa1bbf189f14d725f7e99a
Normal-3261.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3261.png,Normal/images/Normal-3261.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3261.png,Normal/masks/Normal-3261.png,True,False,299,299,L,137.1,edb1dd62eae769149639ff7dfd37f310
Normal-3262.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3262.png,Normal/images/Normal-3262.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3262.png,Normal/masks/Normal-3262.png,True,False,299,299,L,127.67,9ea2e013fa01732513fbd8e08cb9a3f2
Normal-3263.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3263.png,Normal/images/Normal-3263.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3263.png,Normal/masks/Normal-3263.png,True,False,299,299,L,105.62,71e9179c08ce2c5164b164cb8a8f5373
Normal-3264.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3264.png,Normal/images/Normal-3264.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3264.png,Normal/masks/Normal-3264.png,True,False,299,299,L,119.32,f521f5573df3baa81ef76f8ca3110270
Normal-3265.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3265.png,Normal/images/Normal-3265.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3265.png,Normal/masks/Normal-3265.png,True,False,299,299,L,161.76,155bfc7b0150ab19ae6bd6ea7482d1f0
Normal-3266.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3266.png,Normal/images/Normal-3266.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3266.png,Normal/masks/Normal-3266.png,True,False,299,299,L,148.64,1a2d949094a2520df0244ca7721eb79b
Normal-3267.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3267.png,Normal/images/Normal-3267.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3267.png,Normal/masks/Normal-3267.png,True,False,299,299,L,121.9,0318d651c9512115343e413e04be77cc
Normal-3268.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3268.png,Normal/images/Normal-3268.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3268.png,Normal/masks/Normal-3268.png,True,False,299,299,L,111.01,af087ca2f3663cb029e42c06b30d7cb4
Normal-3269.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3269.png,Normal/images/Normal-3269.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3269.png,Normal/masks/Normal-3269.png,True,False,299,299,L,112.41,35ba23b864d8c94484bbed6c3cf0b199
Normal-327.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-327.png,Normal/images/Normal-327.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-327.png,Normal/masks/Normal-327.png,True,False,299,299,L,124.83,0275abe971dcb00429da164397ff4b1e
Normal-3270.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3270.png,Normal/images/Normal-3270.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3270.png,Normal/masks/Normal-3270.png,True,False,299,299,L,107.3,a43f70dc5b12f250df0e070d3d7c3ed5
Normal-3271.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3271.png,Normal/images/Normal-3271.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3271.png,Normal/masks/Normal-3271.png,True,False,299,299,L,153.36,4e655640f78a64e5a170b1ea51de411e
Normal-3272.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3272.png,Normal/images/Normal-3272.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3272.png,Normal/masks/Normal-3272.png,True,False,299,299,L,138.65,a68edb69965c7f0adf9d72ff11a29392
Normal-3273.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3273.png,Normal/images/Normal-3273.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3273.png,Normal/masks/Normal-3273.png,True,False,299,299,L,144.83,3c4cc47de1e251c4afe8a7229ea04135
Normal-3274.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3274.png,Normal/images/Normal-3274.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3274.png,Normal/masks/Normal-3274.png,True,False,299,299,L,159.08,e9f6a6d14f864492c5045b950963015e
Normal-3275.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3275.png,Normal/images/Normal-3275.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3275.png,Normal/masks/Normal-3275.png,True,False,299,299,L,117.76,8b94f46de52694ed6a6ddc5f44dd5b75
Normal-3276.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3276.png,Normal/images/Normal-3276.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3276.png,Normal/masks/Normal-3276.png,True,False,299,299,L,141.33,2d726931196414a7eab858dfab4c02ee
Normal-3277.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3277.png,Normal/images/Normal-3277.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3277.png,Normal/masks/Normal-3277.png,True,False,299,299,L,121.52,019281abb833b7494dbaf145636c8ca1
Normal-3278.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3278.png,Normal/images/Normal-3278.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3278.png,Normal/masks/Normal-3278.png,True,False,299,299,L,114.41,3f3f4b0f10a6eb16d4a08b626e717877
Normal-3279.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3279.png,Normal/images/Normal-3279.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3279.png,Normal/masks/Normal-3279.png,True,False,299,299,L,106.04,c2e2b248b0e9471c9771a21d806de09f
Normal-328.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-328.png,Normal/images/Normal-328.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-328.png,Normal/masks/Normal-328.png,True,False,299,299,L,128.51,780de8b64567d59140711788575681a5
Normal-3280.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3280.png,Normal/images/Normal-3280.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3280.png,Normal/masks/Normal-3280.png,True,False,299,299,L,126.73,eaa9a3a1608217b39e7197904a4b6860
Normal-3281.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3281.png,Normal/images/Normal-3281.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3281.png,Normal/masks/Normal-3281.png,True,False,299,299,L,152.82,934460ffa03ed00d18d7023591147b86
Normal-3282.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3282.png,Normal/images/Normal-3282.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3282.png,Normal/masks/Normal-3282.png,True,False,299,299,L,114.46,01a42dc7c29b6eee69813769b4c413a3
Normal-3283.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3283.png,Normal/images/Normal-3283.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3283.png,Normal/masks/Normal-3283.png,True,False,299,299,L,107.17,815fdf259ab5caa4485b564904a236bb
Normal-3284.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3284.png,Normal/images/Normal-3284.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3284.png,Normal/masks/Normal-3284.png,True,False,299,299,L,159.28,11bc4fe33444b060671dc99a65539d4f
Normal-3285.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3285.png,Normal/images/Normal-3285.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3285.png,Normal/masks/Normal-3285.png,True,False,299,299,L,112.63,4c3b3e1b42d9323bf5207eae41b32b0e
Normal-3286.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3286.png,Normal/images/Normal-3286.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3286.png,Normal/masks/Normal-3286.png,True,False,299,299,L,99.6,5fbb5b7b0ac7a081a99f923b93746af7
Normal-3287.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3287.png,Normal/images/Normal-3287.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3287.png,Normal/masks/Normal-3287.png,True,False,299,299,L,105.47,02601785e46ce5958f76e623dba93379
Normal-3288.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3288.png,Normal/images/Normal-3288.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3288.png,Normal/masks/Normal-3288.png,True,False,299,299,L,152.53,88d98b5b45b11e6834d4a7dbd865401c
Normal-3289.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3289.png,Normal/images/Normal-3289.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3289.png,Normal/masks/Normal-3289.png,True,False,299,299,L,141.22,6ec4523217b26f12ca8e2309f9619c24
Normal-329.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-329.png,Normal/images/Normal-329.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-329.png,Normal/masks/Normal-329.png,True,False,299,299,L,106.91,9499f363bb8fdb6971733f0de0dacd49
Normal-3290.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3290.png,Normal/images/Normal-3290.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3290.png,Normal/masks/Normal-3290.png,True,False,299,299,L,132.06,95551dbc6735cfcbe3ca37f18bee7925
Normal-3291.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3291.png,Normal/images/Normal-3291.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3291.png,Normal/masks/Normal-3291.png,True,False,299,299,L,174.37,bb2ddadc5b921ef141edb853d0b1ed4d
Normal-3292.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3292.png,Normal/images/Normal-3292.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3292.png,Normal/masks/Normal-3292.png,True,False,299,299,L,119.28,0893405aa9cd26c12708976619772570
Normal-3293.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3293.png,Normal/images/Normal-3293.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3293.png,Normal/masks/Normal-3293.png,True,False,299,299,L,101.56,c6cb17cfad75979a6cf9d0f7403b9b0d
Normal-3294.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3294.png,Normal/images/Normal-3294.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3294.png,Normal/masks/Normal-3294.png,True,False,299,299,L,151.01,7cf91ffbf39daf7d210bcff7ac1a1f79
Normal-3295.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3295.png,Normal/images/Normal-3295.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3295.png,Normal/masks/Normal-3295.png,True,False,299,299,L,105.4,ff99361e6d372531370229ce30adef43
Normal-3296.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3296.png,Normal/images/Normal-3296.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3296.png,Normal/masks/Normal-3296.png,True,False,299,299,L,133.75,fba1e07cfe57065da6954d534fc9441a
Normal-3297.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3297.png,Normal/images/Normal-3297.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3297.png,Normal/masks/Normal-3297.png,True,False,299,299,L,144.09,09b8aa261c51243452c3b6c76de6ba6e
Normal-3298.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3298.png,Normal/images/Normal-3298.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3298.png,Normal/masks/Normal-3298.png,True,False,299,299,L,169.83,9d5a531304507d95396ad29da479a6e6
Normal-3299.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3299.png,Normal/images/Normal-3299.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3299.png,Normal/masks/Normal-3299.png,True,False,299,299,L,99.8,07ceb848346496b5458b850843014295
Normal-33.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-33.png,Normal/images/Normal-33.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-33.png,Normal/masks/Normal-33.png,True,False,299,299,L,124.34,9c36a52469c75d5d85c77adb76e064e1
Normal-330.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-330.png,Normal/images/Normal-330.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-330.png,Normal/masks/Normal-330.png,True,False,299,299,L,105.18,c6076577da832d1b6e6cbb565bd4110e
Normal-3300.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3300.png,Normal/images/Normal-3300.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3300.png,Normal/masks/Normal-3300.png,True,False,299,299,L,119.37,20047338a75680c5a4c1b1f9e6924db7
Normal-3301.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3301.png,Normal/images/Normal-3301.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3301.png,Normal/masks/Normal-3301.png,True,False,299,299,L,158.03,1c06141b3a8d4f05437edad2b017fe31
Normal-3302.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3302.png,Normal/images/Normal-3302.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3302.png,Normal/masks/Normal-3302.png,True,False,299,299,L,104.66,6cbd283dc676059292ad68c628a36be3
Normal-3303.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3303.png,Normal/images/Normal-3303.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3303.png,Normal/masks/Normal-3303.png,True,False,299,299,L,147.89,631001debd99b14f6ba0107b45f0ca09
Normal-3304.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3304.png,Normal/images/Normal-3304.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3304.png,Normal/masks/Normal-3304.png,True,False,299,299,L,111.79,687f9ab5fa063d4a4a80d7ebe1c4578c
Normal-3305.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3305.png,Normal/images/Normal-3305.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3305.png,Normal/masks/Normal-3305.png,True,False,299,299,L,126.74,d8da8a4f07e07df52447ce3faca02ebc
Normal-3306.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3306.png,Normal/images/Normal-3306.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3306.png,Normal/masks/Normal-3306.png,True,False,299,299,L,112.3,0d721d6895ca313dc664833b1c78c5b4
Normal-3307.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3307.png,Normal/images/Normal-3307.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3307.png,Normal/masks/Normal-3307.png,True,False,299,299,L,99.51,3d8fc7f75c3636a58c559ecc842985d9
Normal-3308.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3308.png,Normal/images/Normal-3308.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3308.png,Normal/masks/Normal-3308.png,True,False,299,299,L,127.4,55e39f53334dcf304ee87273e7e3c642
Normal-3309.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3309.png,Normal/images/Normal-3309.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3309.png,Normal/masks/Normal-3309.png,True,False,299,299,L,108.76,a8e0875be53d3e434757b615696f2521
Normal-331.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-331.png,Normal/images/Normal-331.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-331.png,Normal/masks/Normal-331.png,True,False,299,299,L,124.75,2ba04d688e860aa21243687d88bebbf2
Normal-3310.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3310.png,Normal/images/Normal-3310.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3310.png,Normal/masks/Normal-3310.png,True,False,299,299,L,113.6,f6157e9fe9823cf690c97a5a13880907
Normal-3311.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3311.png,Normal/images/Normal-3311.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3311.png,Normal/masks/Normal-3311.png,True,False,299,299,L,147.77,dc6f50cb5e25a996d283cec8e82ada58
Normal-3312.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3312.png,Normal/images/Normal-3312.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3312.png,Normal/masks/Normal-3312.png,True,False,299,299,L,114.61,6d3433b9385bcfd9f47cc5f0432126f6
Normal-3313.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3313.png,Normal/images/Normal-3313.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3313.png,Normal/masks/Normal-3313.png,True,False,299,299,L,111.7,6d078c94666ff4d1a3d0360a48780fd1
Normal-3314.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3314.png,Normal/images/Normal-3314.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3314.png,Normal/masks/Normal-3314.png,True,False,299,299,L,157.46,9f390c160bd00a65a5b3664b9a2080a0
Normal-3315.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3315.png,Normal/images/Normal-3315.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3315.png,Normal/masks/Normal-3315.png,True,False,299,299,L,180.08,f2d148e3e505918ee17321207c8b686a
Normal-3316.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3316.png,Normal/images/Normal-3316.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3316.png,Normal/masks/Normal-3316.png,True,False,299,299,L,107.82,92c0d34c049743591d911a925c579587
Normal-3317.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3317.png,Normal/images/Normal-3317.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3317.png,Normal/masks/Normal-3317.png,True,False,299,299,L,169.67,2c49d7b67af496d4d166a7b7cfdc0bbb
Normal-3318.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3318.png,Normal/images/Normal-3318.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3318.png,Normal/masks/Normal-3318.png,True,False,299,299,L,122.43,f8f5da42c373975f5a4c1b2b40bf021e
Normal-3319.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3319.png,Normal/images/Normal-3319.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3319.png,Normal/masks/Normal-3319.png,True,False,299,299,L,124.95,96f424b4a6d76caedfa62ea8a3115a68
Normal-332.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-332.png,Normal/images/Normal-332.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-332.png,Normal/masks/Normal-332.png,True,False,299,299,L,130.24,21ac3eef7dbcf923d0ef2d7e1b0fd839
Normal-3320.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3320.png,Normal/images/Normal-3320.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3320.png,Normal/masks/Normal-3320.png,True,False,299,299,L,126.41,32cc867c7e0105581ecd9b5f592cd3a6
Normal-3321.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3321.png,Normal/images/Normal-3321.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3321.png,Normal/masks/Normal-3321.png,True,False,299,299,L,115.45,6053ef364de4ed6f5f381dce0625ccc8
Normal-3322.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3322.png,Normal/images/Normal-3322.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3322.png,Normal/masks/Normal-3322.png,True,False,299,299,L,127.58,d7eb69f944944a0891f48043b8ce37c9
Normal-3323.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3323.png,Normal/images/Normal-3323.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3323.png,Normal/masks/Normal-3323.png,True,False,299,299,L,137.7,82658399a802943912cc4278374adccd
Normal-3324.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3324.png,Normal/images/Normal-3324.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3324.png,Normal/masks/Normal-3324.png,True,False,299,299,L,109.38,3d0bf4c03720e717fbf2867173d27399
Normal-3325.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3325.png,Normal/images/Normal-3325.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3325.png,Normal/masks/Normal-3325.png,True,False,299,299,L,132.08,76fca72433cdae3d0e41a14430c2ca97
Normal-3326.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3326.png,Normal/images/Normal-3326.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3326.png,Normal/masks/Normal-3326.png,True,False,299,299,L,150.05,718404f19a9a82039c5826abb61aedf5
Normal-3327.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3327.png,Normal/images/Normal-3327.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3327.png,Normal/masks/Normal-3327.png,True,False,299,299,L,105.74,4e7b8a521f57317d3a1aa92bcb840e35
Normal-3328.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3328.png,Normal/images/Normal-3328.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3328.png,Normal/masks/Normal-3328.png,True,False,299,299,L,174.72,aebaa69f6a2f498cb19b96f6b9769ba8
Normal-3329.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3329.png,Normal/images/Normal-3329.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3329.png,Normal/masks/Normal-3329.png,True,False,299,299,L,106.57,b37c93db25c1c7705d004ef15bc01b41
Normal-333.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-333.png,Normal/images/Normal-333.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-333.png,Normal/masks/Normal-333.png,True,False,299,299,L,139.31,ee93b2c28cde5f16845a91aa36ea8f4a
Normal-3330.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3330.png,Normal/images/Normal-3330.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3330.png,Normal/masks/Normal-3330.png,True,False,299,299,L,132.55,850a53dba3ffa7063b786f20c2c40737
Normal-3331.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3331.png,Normal/images/Normal-3331.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3331.png,Normal/masks/Normal-3331.png,True,False,299,299,L,133.61,5e62562aa67f5b1b1a64f6b42d45b804
Normal-3332.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3332.png,Normal/images/Normal-3332.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3332.png,Normal/masks/Normal-3332.png,True,False,299,299,L,106.59,91e21586b05004f7a865dc256ae66bcf
Normal-3333.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3333.png,Normal/images/Normal-3333.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3333.png,Normal/masks/Normal-3333.png,True,False,299,299,L,170.26,1af45358b6dd81e884688d980c2e199e
Normal-3334.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3334.png,Normal/images/Normal-3334.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3334.png,Normal/masks/Normal-3334.png,True,False,299,299,L,107.81,ab679ff1475965690338308703cb3194
Normal-3335.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3335.png,Normal/images/Normal-3335.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3335.png,Normal/masks/Normal-3335.png,True,False,299,299,L,131.85,4774ab1d0f86b2b03304df356f49fa9a
Normal-3336.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3336.png,Normal/images/Normal-3336.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3336.png,Normal/masks/Normal-3336.png,True,False,299,299,L,145.86,cc07d81aeb6c90762a5547e878f916f6
Normal-3337.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3337.png,Normal/images/Normal-3337.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3337.png,Normal/masks/Normal-3337.png,True,False,299,299,L,129.71,56e0e8a6427a537d7d9d8b9a913921aa
Normal-3338.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3338.png,Normal/images/Normal-3338.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3338.png,Normal/masks/Normal-3338.png,True,False,299,299,L,130.24,3847cb761a73f371974317c4036d5bc2
Normal-3339.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3339.png,Normal/images/Normal-3339.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3339.png,Normal/masks/Normal-3339.png,True,False,299,299,L,162.2,e82fc9b398dcc6c1bd078a8895cf1376
Normal-334.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-334.png,Normal/images/Normal-334.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-334.png,Normal/masks/Normal-334.png,True,False,299,299,L,145.05,0a508d07b98e2dea639984521627873e
Normal-3340.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3340.png,Normal/images/Normal-3340.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3340.png,Normal/masks/Normal-3340.png,True,False,299,299,L,106.68,66d0a25c5e3e52d8c92478fdff2f8c5b
Normal-3341.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3341.png,Normal/images/Normal-3341.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3341.png,Normal/masks/Normal-3341.png,True,False,299,299,L,123.92,535f23e4502097c51fb8ac512d610fc6
Normal-3342.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3342.png,Normal/images/Normal-3342.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3342.png,Normal/masks/Normal-3342.png,True,False,299,299,L,154.11,0a8a386b059849330338aedc7e00fe27
Normal-3343.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3343.png,Normal/images/Normal-3343.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3343.png,Normal/masks/Normal-3343.png,True,False,299,299,L,122.45,e2e23991ff0ee2e7ea3b1180de8a04a4
Normal-3344.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3344.png,Normal/images/Normal-3344.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3344.png,Normal/masks/Normal-3344.png,True,False,299,299,L,147.47,3fe61ec1cc300f73f6fdc6c9c4e61233
Normal-3345.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3345.png,Normal/images/Normal-3345.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3345.png,Normal/masks/Normal-3345.png,True,False,299,299,L,130.06,907251a0ffd436b6f55cb05d76b111b8
Normal-3346.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3346.png,Normal/images/Normal-3346.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3346.png,Normal/masks/Normal-3346.png,True,False,299,299,L,138.29,161066e89bfa4ee24507386aaeb3e7ec
Normal-3347.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3347.png,Normal/images/Normal-3347.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3347.png,Normal/masks/Normal-3347.png,True,False,299,299,L,119.11,be48a29855fd13236e8a79a10e4f8b33
Normal-3348.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3348.png,Normal/images/Normal-3348.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3348.png,Normal/masks/Normal-3348.png,True,False,299,299,L,157.99,acf378a3876675f6bd0870b8ef5df0a2
Normal-3349.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3349.png,Normal/images/Normal-3349.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3349.png,Normal/masks/Normal-3349.png,True,False,299,299,L,113.06,cd49ab84db64b45109720615e7a86cf4
Normal-335.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-335.png,Normal/images/Normal-335.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-335.png,Normal/masks/Normal-335.png,True,False,299,299,L,132.05,615a8f7bfc9003a370f5d057796cde7b
Normal-3350.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3350.png,Normal/images/Normal-3350.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3350.png,Normal/masks/Normal-3350.png,True,False,299,299,L,147.89,be75fdb428e8647c34bf07a1e0d63c94
Normal-3351.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3351.png,Normal/images/Normal-3351.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3351.png,Normal/masks/Normal-3351.png,True,False,299,299,L,69.84,2de17135783ac4b94db77a005e3275cd
Normal-3352.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3352.png,Normal/images/Normal-3352.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3352.png,Normal/masks/Normal-3352.png,True,False,299,299,L,123.77,6b67b7936a1469c5e17d3ed96c16b414
Normal-3353.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3353.png,Normal/images/Normal-3353.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3353.png,Normal/masks/Normal-3353.png,True,False,299,299,L,173.96,68b9f6dc6de61e164584a880e9adc004
Normal-3354.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3354.png,Normal/images/Normal-3354.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3354.png,Normal/masks/Normal-3354.png,True,False,299,299,L,135.07,521d8d711010b1f53eadd1e937668c61
Normal-3355.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3355.png,Normal/images/Normal-3355.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3355.png,Normal/masks/Normal-3355.png,True,False,299,299,L,179.95,1800f64c0df04d21d91ecb39b7b5cf38
Normal-3356.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3356.png,Normal/images/Normal-3356.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3356.png,Normal/masks/Normal-3356.png,True,False,299,299,L,116.18,2fa0f2c3add918f4e2a419d36b8e2e36
Normal-3357.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3357.png,Normal/images/Normal-3357.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3357.png,Normal/masks/Normal-3357.png,True,False,299,299,L,144.3,18c9d90f8141845e179ec3c327bb8a7d
Normal-3358.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3358.png,Normal/images/Normal-3358.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3358.png,Normal/masks/Normal-3358.png,True,False,299,299,L,147.3,4ab4706c78d1626d8bbb2eab5d53ed46
Normal-3359.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3359.png,Normal/images/Normal-3359.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3359.png,Normal/masks/Normal-3359.png,True,False,299,299,L,100.73,047b8a2c0337067a5d800598b7ca8b36
Normal-336.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-336.png,Normal/images/Normal-336.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-336.png,Normal/masks/Normal-336.png,True,False,299,299,L,132.91,3f2438a9b1498f8b4deb9249e2d14e74
Normal-3360.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3360.png,Normal/images/Normal-3360.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3360.png,Normal/masks/Normal-3360.png,True,False,299,299,L,130.36,09a0c2a9bd8410d981f17773b17e6ffe
Normal-3361.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3361.png,Normal/images/Normal-3361.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3361.png,Normal/masks/Normal-3361.png,True,False,299,299,L,150.04,4bca837deb1833490937f94b42d322a2
Normal-3362.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3362.png,Normal/images/Normal-3362.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3362.png,Normal/masks/Normal-3362.png,True,False,299,299,L,137.96,a099e103b807b269ed80aa0bdf8b5d4d
Normal-3363.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3363.png,Normal/images/Normal-3363.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3363.png,Normal/masks/Normal-3363.png,True,False,299,299,L,130.75,1d527bd90445a987d4c04f184390d78e
Normal-3364.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3364.png,Normal/images/Normal-3364.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3364.png,Normal/masks/Normal-3364.png,True,False,299,299,L,106.7,87bdf739c6b2f531bcd8b3a06b331e23
Normal-3365.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3365.png,Normal/images/Normal-3365.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3365.png,Normal/masks/Normal-3365.png,True,False,299,299,L,139.38,a8d0dd3efe6ebd959159f9830a8e221e
Normal-3366.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3366.png,Normal/images/Normal-3366.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3366.png,Normal/masks/Normal-3366.png,True,False,299,299,L,114.8,6b5a74d94ec3474a03c85e5db68ac653
Normal-3367.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3367.png,Normal/images/Normal-3367.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3367.png,Normal/masks/Normal-3367.png,True,False,299,299,L,118.28,4e660293dfcfe73032a1f89b26750904
Normal-3368.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3368.png,Normal/images/Normal-3368.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3368.png,Normal/masks/Normal-3368.png,True,False,299,299,L,116.53,bfc5ef3e0c94b858c45e20f9f7eb2a9e
Normal-3369.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3369.png,Normal/images/Normal-3369.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3369.png,Normal/masks/Normal-3369.png,True,False,299,299,L,110.74,75c4bd380fbac7ff366d0e207cf69da4
Normal-337.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-337.png,Normal/images/Normal-337.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-337.png,Normal/masks/Normal-337.png,True,False,299,299,L,126.03,da90f320b7e8952f9c4426abd9afff21
Normal-3370.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3370.png,Normal/images/Normal-3370.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3370.png,Normal/masks/Normal-3370.png,True,False,299,299,L,160.13,ff31652a336d9822e42c49776c70e2c7
Normal-3371.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3371.png,Normal/images/Normal-3371.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3371.png,Normal/masks/Normal-3371.png,True,False,299,299,L,117.74,93077b02737fbd6a21f50b9804a0ceca
Normal-3372.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3372.png,Normal/images/Normal-3372.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3372.png,Normal/masks/Normal-3372.png,True,False,299,299,L,140.12,bbed8cc053c350c0a9d8dde82c64ef02
Normal-3373.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3373.png,Normal/images/Normal-3373.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3373.png,Normal/masks/Normal-3373.png,True,False,299,299,L,128.65,8e066e10b6a5734d62c2ec07e5a7db7b
Normal-3374.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3374.png,Normal/images/Normal-3374.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3374.png,Normal/masks/Normal-3374.png,True,False,299,299,L,97.14,8278d53dd0a5d20ce53dd47ea0fcadbf
Normal-3375.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3375.png,Normal/images/Normal-3375.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3375.png,Normal/masks/Normal-3375.png,True,False,299,299,L,114.78,a1979ea483d7ca671791a0b3d609999a
Normal-3376.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3376.png,Normal/images/Normal-3376.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3376.png,Normal/masks/Normal-3376.png,True,False,299,299,L,169.19,fce9506f3080025bad7509283adce2c4
Normal-3377.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3377.png,Normal/images/Normal-3377.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3377.png,Normal/masks/Normal-3377.png,True,False,299,299,L,118.48,e1912e13028cb03d744f1d76f0482847
Normal-3378.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3378.png,Normal/images/Normal-3378.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3378.png,Normal/masks/Normal-3378.png,True,False,299,299,L,121.82,0a4f629d6d3b950de0f4a2c2f81f3a7b
Normal-3379.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3379.png,Normal/images/Normal-3379.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3379.png,Normal/masks/Normal-3379.png,True,False,299,299,L,103.62,c79700a96d7be8d5ab85b2e23394fa7f
Normal-338.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-338.png,Normal/images/Normal-338.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-338.png,Normal/masks/Normal-338.png,True,False,299,299,L,126.04,37bcae180ee3d24dffcac861c8880fb1
Normal-3380.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3380.png,Normal/images/Normal-3380.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3380.png,Normal/masks/Normal-3380.png,True,False,299,299,L,124.26,72c5b12fc1512fa886d5fe97cd01faa9
Normal-3381.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3381.png,Normal/images/Normal-3381.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3381.png,Normal/masks/Normal-3381.png,True,False,299,299,L,165.04,a2d901f9035fc1a366fe6ef390a42025
Normal-3382.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3382.png,Normal/images/Normal-3382.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3382.png,Normal/masks/Normal-3382.png,True,False,299,299,L,103.34,c786fa832cafba8f5f252ffe9571b640
Normal-3383.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3383.png,Normal/images/Normal-3383.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3383.png,Normal/masks/Normal-3383.png,True,False,299,299,L,119.14,0b058bf17f485749a308bc22a021a89b
Normal-3384.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3384.png,Normal/images/Normal-3384.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3384.png,Normal/masks/Normal-3384.png,True,False,299,299,L,107.02,c4e5069d59e5ea4ebe76e9ec0a38c381
Normal-3385.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3385.png,Normal/images/Normal-3385.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3385.png,Normal/masks/Normal-3385.png,True,False,299,299,L,121.59,1ff0497b0bee2f71d82577193148e876
Normal-3386.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3386.png,Normal/images/Normal-3386.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3386.png,Normal/masks/Normal-3386.png,True,False,299,299,L,105.81,363c4f688a948c60195b6349d03aab3a
Normal-3387.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3387.png,Normal/images/Normal-3387.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3387.png,Normal/masks/Normal-3387.png,True,False,299,299,L,147.4,12e55ebc02babb6811e376aaeed86d1f
Normal-3388.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3388.png,Normal/images/Normal-3388.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3388.png,Normal/masks/Normal-3388.png,True,False,299,299,L,118.93,de775f0bd2c472257ec2ff9a1e456f24
Normal-3389.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3389.png,Normal/images/Normal-3389.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3389.png,Normal/masks/Normal-3389.png,True,False,299,299,L,115.34,ec15fb6fcbe212b2389627b7d40ce08d
Normal-339.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-339.png,Normal/images/Normal-339.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-339.png,Normal/masks/Normal-339.png,True,False,299,299,L,125.82,5efc2aeabb30ac84c64840d23063e11e
Normal-3390.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3390.png,Normal/images/Normal-3390.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3390.png,Normal/masks/Normal-3390.png,True,False,299,299,L,117.0,828c235bc0f8d6b3b615365898680bc1
Normal-3391.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3391.png,Normal/images/Normal-3391.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3391.png,Normal/masks/Normal-3391.png,True,False,299,299,L,94.0,fac994d5525b356c1585131b3b592308
Normal-3392.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3392.png,Normal/images/Normal-3392.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3392.png,Normal/masks/Normal-3392.png,True,False,299,299,L,134.27,bfdc89f046c389e7f712aeca41b99b38
Normal-3393.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3393.png,Normal/images/Normal-3393.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3393.png,Normal/masks/Normal-3393.png,True,False,299,299,L,166.2,e7d5615a50c253a9e68d268738c63539
Normal-3394.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3394.png,Normal/images/Normal-3394.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3394.png,Normal/masks/Normal-3394.png,True,False,299,299,L,150.36,0628fefe8e602affc5c64da6e7d15d2a
Normal-3395.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3395.png,Normal/images/Normal-3395.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3395.png,Normal/masks/Normal-3395.png,True,False,299,299,L,117.16,1ee0435e3f4cbc7d278d76984435b192
Normal-3396.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3396.png,Normal/images/Normal-3396.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3396.png,Normal/masks/Normal-3396.png,True,False,299,299,L,116.95,5a94555163c1495119f8a058a5fa521f
Normal-3397.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3397.png,Normal/images/Normal-3397.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3397.png,Normal/masks/Normal-3397.png,True,False,299,299,L,104.28,0e02d9cc2c3ba7c4a3e58793e8027682
Normal-3398.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3398.png,Normal/images/Normal-3398.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3398.png,Normal/masks/Normal-3398.png,True,False,299,299,L,168.74,1b293557f8c65f4719a2b3b966a68560
Normal-3399.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3399.png,Normal/images/Normal-3399.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3399.png,Normal/masks/Normal-3399.png,True,False,299,299,L,143.24,0493729928e603d6e673d380bcb1963d
Normal-34.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-34.png,Normal/images/Normal-34.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-34.png,Normal/masks/Normal-34.png,True,False,299,299,L,104.32,af99ccbccdfd1c798337014f5a2aff9f
Normal-340.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-340.png,Normal/images/Normal-340.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-340.png,Normal/masks/Normal-340.png,True,False,299,299,L,128.49,f5fe1a666481c481aeb97a54fc2f8dd7
Normal-3400.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3400.png,Normal/images/Normal-3400.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3400.png,Normal/masks/Normal-3400.png,True,False,299,299,L,117.83,a67cd3ebb969f0a0495d46e1332927ee
Normal-3401.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3401.png,Normal/images/Normal-3401.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3401.png,Normal/masks/Normal-3401.png,True,False,299,299,L,154.71,02c9e42431deef431eac623b4e12033b
Normal-3402.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3402.png,Normal/images/Normal-3402.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3402.png,Normal/masks/Normal-3402.png,True,False,299,299,L,98.93,bebf46936a9f5d9a3c790e9ed40f3172
Normal-3403.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3403.png,Normal/images/Normal-3403.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3403.png,Normal/masks/Normal-3403.png,True,False,299,299,L,109.79,1882feb2c680a1e502c8090fe1de5d68
Normal-3404.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3404.png,Normal/images/Normal-3404.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3404.png,Normal/masks/Normal-3404.png,True,False,299,299,L,109.25,03d1246ddebaa7fc68e7fe3573cd9fa6
Normal-3405.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3405.png,Normal/images/Normal-3405.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3405.png,Normal/masks/Normal-3405.png,True,False,299,299,L,146.3,ac179984507bb4022957eb7766e40f24
Normal-3406.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3406.png,Normal/images/Normal-3406.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3406.png,Normal/masks/Normal-3406.png,True,False,299,299,L,161.05,d0de5c26b07c0b48d32ad5a2ab6df747
Normal-3407.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3407.png,Normal/images/Normal-3407.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3407.png,Normal/masks/Normal-3407.png,True,False,299,299,L,180.83,1003e4718e481f4fc2469e6593ace04b
Normal-3408.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3408.png,Normal/images/Normal-3408.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3408.png,Normal/masks/Normal-3408.png,True,False,299,299,L,157.9,fa475de193d2ab61b218f13639eea47c
Normal-3409.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3409.png,Normal/images/Normal-3409.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3409.png,Normal/masks/Normal-3409.png,True,False,299,299,L,117.48,b41f4b7d46b214c67f564cb8ade52dc8
Normal-341.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-341.png,Normal/images/Normal-341.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-341.png,Normal/masks/Normal-341.png,True,False,299,299,L,126.78,e8589a535b3c9914b76585ec6b75cdb0
Normal-3410.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3410.png,Normal/images/Normal-3410.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3410.png,Normal/masks/Normal-3410.png,True,False,299,299,L,98.15,011079322e3dc03d445540fa83a45629
Normal-3411.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3411.png,Normal/images/Normal-3411.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3411.png,Normal/masks/Normal-3411.png,True,False,299,299,L,111.17,7c9bcc11cfdb915f51e557f44103916d
Normal-3412.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3412.png,Normal/images/Normal-3412.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3412.png,Normal/masks/Normal-3412.png,True,False,299,299,L,156.27,a55f8bcddf79c04a64638b6decfa4fb6
Normal-3413.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3413.png,Normal/images/Normal-3413.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3413.png,Normal/masks/Normal-3413.png,True,False,299,299,L,176.58,8275fdf2c9f4b3a4f458becb0b75e46c
Normal-3414.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3414.png,Normal/images/Normal-3414.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3414.png,Normal/masks/Normal-3414.png,True,False,299,299,L,113.74,5e51fec9783541d696ce753b08471f5f
Normal-3415.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3415.png,Normal/images/Normal-3415.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3415.png,Normal/masks/Normal-3415.png,True,False,299,299,L,107.99,3cf61409060625b9c2983d59a4995c17
Normal-3416.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3416.png,Normal/images/Normal-3416.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3416.png,Normal/masks/Normal-3416.png,True,False,299,299,L,116.16,e5eba05231602c041e2fd9dc6f30ace1
Normal-3417.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3417.png,Normal/images/Normal-3417.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3417.png,Normal/masks/Normal-3417.png,True,False,299,299,L,100.06,317b9306a904a862a425be815ba7ab49
Normal-3418.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3418.png,Normal/images/Normal-3418.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3418.png,Normal/masks/Normal-3418.png,True,False,299,299,L,119.47,70d39f4aa9a40923be833089301d89da
Normal-3419.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3419.png,Normal/images/Normal-3419.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3419.png,Normal/masks/Normal-3419.png,True,False,299,299,L,119.66,a0dc658dc468ffb0138e75b7c5ec6fbd
Normal-342.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-342.png,Normal/images/Normal-342.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-342.png,Normal/masks/Normal-342.png,True,False,299,299,L,99.12,958e5b5b8eba72a55c98dfd45a9532c3
Normal-3420.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3420.png,Normal/images/Normal-3420.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3420.png,Normal/masks/Normal-3420.png,True,False,299,299,L,106.1,b1af833b2f80cb77ba29c73fd1aeaa73
Normal-3421.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3421.png,Normal/images/Normal-3421.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3421.png,Normal/masks/Normal-3421.png,True,False,299,299,L,129.27,66272669ba52ee07646f304fd58fe2af
Normal-3422.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3422.png,Normal/images/Normal-3422.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3422.png,Normal/masks/Normal-3422.png,True,False,299,299,L,109.69,04aaaf2ddc8cd3afba0e8024916e7c02
Normal-3423.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3423.png,Normal/images/Normal-3423.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3423.png,Normal/masks/Normal-3423.png,True,False,299,299,L,96.37,9d5aed83d066e5a200efc1b5a657487a
Normal-3424.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3424.png,Normal/images/Normal-3424.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3424.png,Normal/masks/Normal-3424.png,True,False,299,299,L,142.67,4c9e5ce96873220846becd019405db67
Normal-3425.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3425.png,Normal/images/Normal-3425.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3425.png,Normal/masks/Normal-3425.png,True,False,299,299,L,123.54,f48183a57da4cba944d5c5d31bb1f6ca
Normal-3426.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3426.png,Normal/images/Normal-3426.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3426.png,Normal/masks/Normal-3426.png,True,False,299,299,L,117.3,b44ede90b0d2dce00d05e6b400de4185
Normal-3427.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3427.png,Normal/images/Normal-3427.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3427.png,Normal/masks/Normal-3427.png,True,False,299,299,L,117.66,acc84cf0dece82d8e3307ab5668a7a7e
Normal-3428.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3428.png,Normal/images/Normal-3428.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3428.png,Normal/masks/Normal-3428.png,True,False,299,299,L,119.09,56e4ed10071f0d5254a359105cba5648
Normal-3429.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3429.png,Normal/images/Normal-3429.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3429.png,Normal/masks/Normal-3429.png,True,False,299,299,L,117.05,1d2232ebaccea644813022a7ec0b15fe
Normal-343.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-343.png,Normal/images/Normal-343.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-343.png,Normal/masks/Normal-343.png,True,False,299,299,L,111.15,c612a21d76409a0444a08a7de7f783ed
Normal-3430.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3430.png,Normal/images/Normal-3430.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3430.png,Normal/masks/Normal-3430.png,True,False,299,299,L,135.98,ed5a07cda6fb302a29171c4fbe2c3ef2
Normal-3431.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3431.png,Normal/images/Normal-3431.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3431.png,Normal/masks/Normal-3431.png,True,False,299,299,L,124.74,6a609ce1d8aab522c82b00a4eaed2ca6
Normal-3432.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3432.png,Normal/images/Normal-3432.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3432.png,Normal/masks/Normal-3432.png,True,False,299,299,L,118.89,7ac2f8093822f8e4cdac808c98782be4
Normal-3433.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3433.png,Normal/images/Normal-3433.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3433.png,Normal/masks/Normal-3433.png,True,False,299,299,L,118.43,df2455c5e615744443347c302cf28991
Normal-3434.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3434.png,Normal/images/Normal-3434.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3434.png,Normal/masks/Normal-3434.png,True,False,299,299,L,131.28,ee0ceb76b70382baacd5b15641c11b16
Normal-3435.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3435.png,Normal/images/Normal-3435.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3435.png,Normal/masks/Normal-3435.png,True,False,299,299,L,113.53,edb6d0429ae06f826b3e55e11ea3a746
Normal-3436.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3436.png,Normal/images/Normal-3436.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3436.png,Normal/masks/Normal-3436.png,True,False,299,299,L,121.31,3c129726761f07232a7f01544ef43974
Normal-3437.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3437.png,Normal/images/Normal-3437.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3437.png,Normal/masks/Normal-3437.png,True,False,299,299,L,148.0,3a6c62d086500a3e04cac798883cedd9
Normal-3438.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3438.png,Normal/images/Normal-3438.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3438.png,Normal/masks/Normal-3438.png,True,False,299,299,L,109.52,aaa4d225c959f2724ec93afa0e04d7d4
Normal-3439.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3439.png,Normal/images/Normal-3439.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3439.png,Normal/masks/Normal-3439.png,True,False,299,299,L,105.98,07df8f906a46e50f0cd60aa5b8e35bac
Normal-344.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-344.png,Normal/images/Normal-344.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-344.png,Normal/masks/Normal-344.png,True,False,299,299,L,117.89,43bfa6f313efdf9ca83a659d36f40bfb
Normal-3440.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3440.png,Normal/images/Normal-3440.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3440.png,Normal/masks/Normal-3440.png,True,False,299,299,L,122.62,e2145106dea5774eded42433d4e17b74
Normal-3441.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3441.png,Normal/images/Normal-3441.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3441.png,Normal/masks/Normal-3441.png,True,False,299,299,L,127.01,ab8b88cc73476f40d287dcc632f2c81d
Normal-3442.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3442.png,Normal/images/Normal-3442.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3442.png,Normal/masks/Normal-3442.png,True,False,299,299,L,121.68,12d6101a1b9258ad9c364c4209ae9420
Normal-3443.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3443.png,Normal/images/Normal-3443.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3443.png,Normal/masks/Normal-3443.png,True,False,299,299,L,109.59,2b7adb7dea8b02db3283c4d09cdf4137
Normal-3444.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3444.png,Normal/images/Normal-3444.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3444.png,Normal/masks/Normal-3444.png,True,False,299,299,L,134.8,3d1db85cfbff3c9532bf8f6135e9fdcd
Normal-3445.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3445.png,Normal/images/Normal-3445.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3445.png,Normal/masks/Normal-3445.png,True,False,299,299,L,97.7,695f99811c863cc595b95ab4deed2ff7
Normal-3446.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3446.png,Normal/images/Normal-3446.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3446.png,Normal/masks/Normal-3446.png,True,False,299,299,L,121.27,b67bda08151a08be142c047f121cbb29
Normal-3447.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3447.png,Normal/images/Normal-3447.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3447.png,Normal/masks/Normal-3447.png,True,False,299,299,L,134.66,26f196964c0b305c97ae74e9c370dbd5
Normal-3448.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3448.png,Normal/images/Normal-3448.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3448.png,Normal/masks/Normal-3448.png,True,False,299,299,L,124.91,4d8552e50586c14a091f28be362118de
Normal-3449.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3449.png,Normal/images/Normal-3449.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3449.png,Normal/masks/Normal-3449.png,True,False,299,299,L,124.6,4b969e339b2638a37ef7b50b92091007
Normal-345.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-345.png,Normal/images/Normal-345.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-345.png,Normal/masks/Normal-345.png,True,False,299,299,L,135.55,f1bf13107d5a15afe95dedfee5e5a609
Normal-3450.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3450.png,Normal/images/Normal-3450.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3450.png,Normal/masks/Normal-3450.png,True,False,299,299,L,108.06,c217f37ccd38f9a0474298841b005f98
Normal-3451.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3451.png,Normal/images/Normal-3451.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3451.png,Normal/masks/Normal-3451.png,True,False,299,299,L,176.35,4627d09246fd4ad8a4ce5bd30c68e901
Normal-3452.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3452.png,Normal/images/Normal-3452.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3452.png,Normal/masks/Normal-3452.png,True,False,299,299,L,106.23,091676d943c163831288ce76a5284813
Normal-3453.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3453.png,Normal/images/Normal-3453.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3453.png,Normal/masks/Normal-3453.png,True,False,299,299,L,142.36,bbe136a8a36c6d13630ac48290911cee
Normal-3454.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3454.png,Normal/images/Normal-3454.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3454.png,Normal/masks/Normal-3454.png,True,False,299,299,L,131.01,06e95d21d98e5b706cd3bf91cd56b5c2
Normal-3455.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3455.png,Normal/images/Normal-3455.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3455.png,Normal/masks/Normal-3455.png,True,False,299,299,L,113.54,aa70e151b2eb075cf121173359076c85
Normal-3456.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3456.png,Normal/images/Normal-3456.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3456.png,Normal/masks/Normal-3456.png,True,False,299,299,L,113.18,61370fa48485062ff717ddcb434cb9cc
Normal-3457.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3457.png,Normal/images/Normal-3457.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3457.png,Normal/masks/Normal-3457.png,True,False,299,299,L,152.86,dc9e1d81089c01aedd77c938755d5251
Normal-3458.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3458.png,Normal/images/Normal-3458.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3458.png,Normal/masks/Normal-3458.png,True,False,299,299,L,139.4,95aae220a9ab07ac3831542edc37a03b
Normal-3459.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3459.png,Normal/images/Normal-3459.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3459.png,Normal/masks/Normal-3459.png,True,False,299,299,L,104.65,d6db620ad5a54c987607a785a94e105a
Normal-346.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-346.png,Normal/images/Normal-346.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-346.png,Normal/masks/Normal-346.png,True,False,299,299,L,125.61,ea3bc73c0e0fa19815fe24304087feb8
Normal-3460.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3460.png,Normal/images/Normal-3460.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3460.png,Normal/masks/Normal-3460.png,True,False,299,299,L,154.14,e9a2d3e54169d89a62e3d1386edc78c7
Normal-3461.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3461.png,Normal/images/Normal-3461.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3461.png,Normal/masks/Normal-3461.png,True,False,299,299,L,166.77,5cc9565786526ae3dc03991e26ed407c
Normal-3462.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3462.png,Normal/images/Normal-3462.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3462.png,Normal/masks/Normal-3462.png,True,False,299,299,L,135.04,503350f216d5f90aea1c7625f81854d2
Normal-3463.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3463.png,Normal/images/Normal-3463.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3463.png,Normal/masks/Normal-3463.png,True,False,299,299,L,163.97,0ff16e100d6ff63e3139d8e044e27632
Normal-3464.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3464.png,Normal/images/Normal-3464.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3464.png,Normal/masks/Normal-3464.png,True,False,299,299,L,107.66,67fa51c689892189c7e5f0fb6ee91039
Normal-3465.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3465.png,Normal/images/Normal-3465.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3465.png,Normal/masks/Normal-3465.png,True,False,299,299,L,122.96,4d6cb1b86e45bed552168c0c1ed1d0e4
Normal-3466.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3466.png,Normal/images/Normal-3466.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3466.png,Normal/masks/Normal-3466.png,True,False,299,299,L,93.64,7f3007fc32e68960aa4bdff4cb1bdd95
Normal-3467.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3467.png,Normal/images/Normal-3467.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3467.png,Normal/masks/Normal-3467.png,True,False,299,299,L,145.91,a9405c75c5ef479e9ce0c6f021c50295
Normal-3468.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3468.png,Normal/images/Normal-3468.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3468.png,Normal/masks/Normal-3468.png,True,False,299,299,L,87.04,31c85043d9701d81a2d75851befb4367
Normal-3469.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3469.png,Normal/images/Normal-3469.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3469.png,Normal/masks/Normal-3469.png,True,False,299,299,L,183.37,1cca96853b6087cd4f42f53796aece1e
Normal-347.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-347.png,Normal/images/Normal-347.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-347.png,Normal/masks/Normal-347.png,True,False,299,299,L,130.3,97915ac0c1649715675f9ddaf3882d08
Normal-3470.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3470.png,Normal/images/Normal-3470.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3470.png,Normal/masks/Normal-3470.png,True,False,299,299,L,180.42,c4ad3d5b999384b3fd4f9d7963d8f1a5
Normal-3471.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3471.png,Normal/images/Normal-3471.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3471.png,Normal/masks/Normal-3471.png,True,False,299,299,L,143.77,b1f56b1f43516cf97480eebfd7b295c6
Normal-3472.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3472.png,Normal/images/Normal-3472.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3472.png,Normal/masks/Normal-3472.png,True,False,299,299,L,134.01,6e5e6005c28e7a3b96edd384fb555232
Normal-3473.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3473.png,Normal/images/Normal-3473.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3473.png,Normal/masks/Normal-3473.png,True,False,299,299,L,121.49,8559fe962fae74c61bdc8eb6a9722cdc
Normal-3474.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3474.png,Normal/images/Normal-3474.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3474.png,Normal/masks/Normal-3474.png,True,False,299,299,L,127.08,6294aab1552f5c9dc06aab496d6e2650
Normal-3475.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3475.png,Normal/images/Normal-3475.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3475.png,Normal/masks/Normal-3475.png,True,False,299,299,L,136.2,700291e51d89a7857388aa3d3aa321a3
Normal-3476.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3476.png,Normal/images/Normal-3476.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3476.png,Normal/masks/Normal-3476.png,True,False,299,299,L,178.06,ea30d4aa8eb952965edc5018fe56c50a
Normal-3477.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3477.png,Normal/images/Normal-3477.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3477.png,Normal/masks/Normal-3477.png,True,False,299,299,L,125.92,1cdba0983ca2d50b3e9f5408c1155af9
Normal-3478.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3478.png,Normal/images/Normal-3478.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3478.png,Normal/masks/Normal-3478.png,True,False,299,299,L,113.44,c7f75934a331e1586e4aa89a15a3155d
Normal-3479.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3479.png,Normal/images/Normal-3479.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3479.png,Normal/masks/Normal-3479.png,True,False,299,299,L,132.81,ea877405af1b638f84ffaf2311c50b10
Normal-348.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-348.png,Normal/images/Normal-348.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-348.png,Normal/masks/Normal-348.png,True,False,299,299,L,121.02,b4136fb8bc5ee0c9e474b655e21ab0d4
Normal-3480.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3480.png,Normal/images/Normal-3480.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3480.png,Normal/masks/Normal-3480.png,True,False,299,299,L,101.32,ca37a9a3485e2f6db3a2c03d640061de
Normal-3481.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3481.png,Normal/images/Normal-3481.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3481.png,Normal/masks/Normal-3481.png,True,False,299,299,L,97.61,bde9b7354155eee76f183f377bf8f5b4
Normal-3482.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3482.png,Normal/images/Normal-3482.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3482.png,Normal/masks/Normal-3482.png,True,False,299,299,L,126.79,72d7803fb28931a076903d3082e36176
Normal-3483.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3483.png,Normal/images/Normal-3483.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3483.png,Normal/masks/Normal-3483.png,True,False,299,299,L,117.4,a20025d9c0d283151a14cd83c08119b4
Normal-3484.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3484.png,Normal/images/Normal-3484.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3484.png,Normal/masks/Normal-3484.png,True,False,299,299,L,138.46,770cac1517aa98a305d3e9ea37f36d02
Normal-3485.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3485.png,Normal/images/Normal-3485.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3485.png,Normal/masks/Normal-3485.png,True,False,299,299,L,111.22,d74970d0c689615f83fec09e41b60f8d
Normal-3486.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3486.png,Normal/images/Normal-3486.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3486.png,Normal/masks/Normal-3486.png,True,False,299,299,L,107.29,bb04fed6be5926758b17394587ae8e6b
Normal-3487.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3487.png,Normal/images/Normal-3487.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3487.png,Normal/masks/Normal-3487.png,True,False,299,299,L,130.17,20f703664bc28c861d04f4a9c88b5c04
Normal-3488.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3488.png,Normal/images/Normal-3488.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3488.png,Normal/masks/Normal-3488.png,True,False,299,299,L,109.2,09f538a22459b9efcfc5998ca55a86b1
Normal-3489.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3489.png,Normal/images/Normal-3489.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3489.png,Normal/masks/Normal-3489.png,True,False,299,299,L,120.12,6598d7615b339f9828533528dca53acb
Normal-349.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-349.png,Normal/images/Normal-349.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-349.png,Normal/masks/Normal-349.png,True,False,299,299,L,135.11,3856ba6f71092150ddb3111caa20bb17
Normal-3490.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3490.png,Normal/images/Normal-3490.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3490.png,Normal/masks/Normal-3490.png,True,False,299,299,L,112.17,295132a49ae6d9b270aaa1253fb752ec
Normal-3491.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3491.png,Normal/images/Normal-3491.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3491.png,Normal/masks/Normal-3491.png,True,False,299,299,L,111.87,a43f2f9268a880b9de8f459240e32162
Normal-3492.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3492.png,Normal/images/Normal-3492.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3492.png,Normal/masks/Normal-3492.png,True,False,299,299,L,116.82,9a306748906cc22126d7e3437351e0c4
Normal-3493.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3493.png,Normal/images/Normal-3493.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3493.png,Normal/masks/Normal-3493.png,True,False,299,299,L,111.64,9047d6c93108d1a4f2c4f09c861cd6b6
Normal-3494.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3494.png,Normal/images/Normal-3494.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3494.png,Normal/masks/Normal-3494.png,True,False,299,299,L,167.76,fe185b20e99b33ff586b5e87a00398ce
Normal-3495.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3495.png,Normal/images/Normal-3495.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3495.png,Normal/masks/Normal-3495.png,True,False,299,299,L,132.22,572c3aaf4be4bc2daf40e91265e7cbcd
Normal-3496.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3496.png,Normal/images/Normal-3496.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3496.png,Normal/masks/Normal-3496.png,True,False,299,299,L,81.95,04f1ef90723726221934ab263cbd7558
Normal-3497.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3497.png,Normal/images/Normal-3497.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3497.png,Normal/masks/Normal-3497.png,True,False,299,299,L,160.13,baee8e12aa30ed76ef51a847b8b1e2db
Normal-3498.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3498.png,Normal/images/Normal-3498.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3498.png,Normal/masks/Normal-3498.png,True,False,299,299,L,110.71,b006e3246f420ca5116fb46c7d0ec1e9
Normal-3499.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3499.png,Normal/images/Normal-3499.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3499.png,Normal/masks/Normal-3499.png,True,False,299,299,L,139.07,ae32e9abf9b87768fd9cff769766eb6b
Normal-35.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-35.png,Normal/images/Normal-35.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-35.png,Normal/masks/Normal-35.png,True,False,299,299,L,130.83,042a755f2cb676a57a89484b1d991fea
Normal-350.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-350.png,Normal/images/Normal-350.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-350.png,Normal/masks/Normal-350.png,True,False,299,299,L,121.09,402a9961092a4ac3bc5b191b4faa6a20
Normal-3500.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3500.png,Normal/images/Normal-3500.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3500.png,Normal/masks/Normal-3500.png,True,False,299,299,L,103.71,0e8ae8119c183a48072a00fdbfdff28b
Normal-3501.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3501.png,Normal/images/Normal-3501.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3501.png,Normal/masks/Normal-3501.png,True,False,299,299,L,176.37,4aba80c133dfb2ceeadf98d0d66a2b7f
Normal-3502.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3502.png,Normal/images/Normal-3502.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3502.png,Normal/masks/Normal-3502.png,True,False,299,299,L,113.73,d78a6960b9702183b1ac383edf82ffac
Normal-3503.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3503.png,Normal/images/Normal-3503.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3503.png,Normal/masks/Normal-3503.png,True,False,299,299,L,132.51,c3d185eaf8c48c294cea2e6616930abb
Normal-3504.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3504.png,Normal/images/Normal-3504.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3504.png,Normal/masks/Normal-3504.png,True,False,299,299,L,118.43,82ef65cf1ade0689941121252f4d8e68
Normal-3505.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3505.png,Normal/images/Normal-3505.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3505.png,Normal/masks/Normal-3505.png,True,False,299,299,L,193.18,35a5f70ca21ce3a919c92806fb848cda
Normal-3506.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3506.png,Normal/images/Normal-3506.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3506.png,Normal/masks/Normal-3506.png,True,False,299,299,L,115.47,48a3f186cd25ebc9bf68b66394239c98
Normal-3507.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3507.png,Normal/images/Normal-3507.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3507.png,Normal/masks/Normal-3507.png,True,False,299,299,L,171.06,24cf282afbb834a511e5a147d0374767
Normal-3508.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3508.png,Normal/images/Normal-3508.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3508.png,Normal/masks/Normal-3508.png,True,False,299,299,L,127.58,a52c55e7565d39d6b331ea6e8e64e6e9
Normal-3509.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3509.png,Normal/images/Normal-3509.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3509.png,Normal/masks/Normal-3509.png,True,False,299,299,L,165.14,2985119e8549e26534a7f6698b14dea2
Normal-351.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-351.png,Normal/images/Normal-351.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-351.png,Normal/masks/Normal-351.png,True,False,299,299,L,144.68,cf55ad0a53cfd485a9197959428432f8
Normal-3510.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3510.png,Normal/images/Normal-3510.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3510.png,Normal/masks/Normal-3510.png,True,False,299,299,L,161.91,59d467f8deaa39125586b6d0ad7e86b8
Normal-3511.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3511.png,Normal/images/Normal-3511.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3511.png,Normal/masks/Normal-3511.png,True,False,299,299,L,112.47,7a1fee8ff8ea02d6683a351991ce1422
Normal-3512.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3512.png,Normal/images/Normal-3512.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3512.png,Normal/masks/Normal-3512.png,True,False,299,299,L,91.66,c1549ff6ac74c8bed489361dddcb0fe4
Normal-3513.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3513.png,Normal/images/Normal-3513.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3513.png,Normal/masks/Normal-3513.png,True,False,299,299,L,162.91,693695a84993c48e10a90d8e039c406f
Normal-3514.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3514.png,Normal/images/Normal-3514.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3514.png,Normal/masks/Normal-3514.png,True,False,299,299,L,117.72,3974c8ba61fa13a8c75275b53fca2bc9
Normal-3515.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3515.png,Normal/images/Normal-3515.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3515.png,Normal/masks/Normal-3515.png,True,False,299,299,L,123.09,c835efba7d970d0ad3abf015b92fd172
Normal-3516.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3516.png,Normal/images/Normal-3516.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3516.png,Normal/masks/Normal-3516.png,True,False,299,299,L,118.92,97fcea52dff8b1b06247f871d88d0ce1
Normal-3517.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3517.png,Normal/images/Normal-3517.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3517.png,Normal/masks/Normal-3517.png,True,False,299,299,L,128.49,a011be04c172ae62cd201ed5424528c5
Normal-3518.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3518.png,Normal/images/Normal-3518.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3518.png,Normal/masks/Normal-3518.png,True,False,299,299,L,125.74,2e8af1fc1b6b00bb2b50063dbed08477
Normal-3519.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3519.png,Normal/images/Normal-3519.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3519.png,Normal/masks/Normal-3519.png,True,False,299,299,L,123.78,83fe1344afd5a23ef7fcbf4b2edae49c
Normal-352.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-352.png,Normal/images/Normal-352.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-352.png,Normal/masks/Normal-352.png,True,False,299,299,L,126.69,7687f813ca926b885d869eb4665206d8
Normal-3520.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3520.png,Normal/images/Normal-3520.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3520.png,Normal/masks/Normal-3520.png,True,False,299,299,L,168.52,bfcd7b222bb8031c38b245222d138a07
Normal-3521.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3521.png,Normal/images/Normal-3521.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3521.png,Normal/masks/Normal-3521.png,True,False,299,299,L,152.27,b28de453a3c8ae0d68d44392f881ef88
Normal-3522.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3522.png,Normal/images/Normal-3522.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3522.png,Normal/masks/Normal-3522.png,True,False,299,299,L,112.06,a30557ac00f944e7b9ff4084dc494ced
Normal-3523.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3523.png,Normal/images/Normal-3523.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3523.png,Normal/masks/Normal-3523.png,True,False,299,299,L,134.35,75f8b39030054d0e3e6e4ff9263ffcfc
Normal-3524.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3524.png,Normal/images/Normal-3524.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3524.png,Normal/masks/Normal-3524.png,True,False,299,299,L,113.21,635a911b6685108e50edf1b8854613d7
Normal-3525.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3525.png,Normal/images/Normal-3525.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3525.png,Normal/masks/Normal-3525.png,True,False,299,299,L,162.18,a8c45d7d7780fabe06bad676de9f0ed0
Normal-3526.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3526.png,Normal/images/Normal-3526.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3526.png,Normal/masks/Normal-3526.png,True,False,299,299,L,109.94,a4fb2990af9cfef846e83a17b367f9be
Normal-3527.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3527.png,Normal/images/Normal-3527.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3527.png,Normal/masks/Normal-3527.png,True,False,299,299,L,112.87,d788517bc04513cc6f60e0fe43fd8678
Normal-3528.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3528.png,Normal/images/Normal-3528.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3528.png,Normal/masks/Normal-3528.png,True,False,299,299,L,179.49,d569b2b91fbb66cb7bbc2df94bcad4e2
Normal-3529.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3529.png,Normal/images/Normal-3529.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3529.png,Normal/masks/Normal-3529.png,True,False,299,299,L,115.23,63c5e3092302782cc68d720bc94179f7
Normal-353.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-353.png,Normal/images/Normal-353.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-353.png,Normal/masks/Normal-353.png,True,False,299,299,L,105.38,e803577736ebc4562c9d5ea0e9399fcd
Normal-3530.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3530.png,Normal/images/Normal-3530.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3530.png,Normal/masks/Normal-3530.png,True,False,299,299,L,140.05,f51538b80dc067eec3a0bd05a8ffe221
Normal-3531.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3531.png,Normal/images/Normal-3531.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3531.png,Normal/masks/Normal-3531.png,True,False,299,299,L,128.3,7ffdc5eba7118f619b621ca68d976cb3
Normal-3532.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3532.png,Normal/images/Normal-3532.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3532.png,Normal/masks/Normal-3532.png,True,False,299,299,L,118.38,329ac1e513d188524c684760db5e2c31
Normal-3533.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3533.png,Normal/images/Normal-3533.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3533.png,Normal/masks/Normal-3533.png,True,False,299,299,L,161.15,de1298fa31257d1b0d5af9353e1ace7d
Normal-3534.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3534.png,Normal/images/Normal-3534.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3534.png,Normal/masks/Normal-3534.png,True,False,299,299,L,134.74,55f4a94622a6775bc99047f76476b5cd
Normal-3535.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3535.png,Normal/images/Normal-3535.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3535.png,Normal/masks/Normal-3535.png,True,False,299,299,L,136.83,cf4a07d90d702ef954258b3872aa68d2
Normal-3536.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3536.png,Normal/images/Normal-3536.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3536.png,Normal/masks/Normal-3536.png,True,False,299,299,L,116.02,924c46c4af5fc9d775560b1a067d1b1d
Normal-3537.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3537.png,Normal/images/Normal-3537.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3537.png,Normal/masks/Normal-3537.png,True,False,299,299,L,122.53,099598c52b5732e51239b90b733dbb24
Normal-3538.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3538.png,Normal/images/Normal-3538.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3538.png,Normal/masks/Normal-3538.png,True,False,299,299,L,103.64,11b0571b63b002b5c0c7f3b93f8f7c8d
Normal-3539.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3539.png,Normal/images/Normal-3539.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3539.png,Normal/masks/Normal-3539.png,True,False,299,299,L,160.01,8e6727a4b7579928383e4a3fb202119a
Normal-354.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-354.png,Normal/images/Normal-354.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-354.png,Normal/masks/Normal-354.png,True,False,299,299,L,129.99,bc2885e8e1406e35d0f8d3cf3a39b0fa
Normal-3540.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3540.png,Normal/images/Normal-3540.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3540.png,Normal/masks/Normal-3540.png,True,False,299,299,L,120.62,7e7091c3b73fc39d208778f04592c70b
Normal-3541.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3541.png,Normal/images/Normal-3541.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3541.png,Normal/masks/Normal-3541.png,True,False,299,299,L,132.98,6b396ca9a0d412441ce738b2ee66cccc
Normal-3542.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3542.png,Normal/images/Normal-3542.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3542.png,Normal/masks/Normal-3542.png,True,False,299,299,L,172.66,3043c970678a0155c377911ec9decf49
Normal-3543.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3543.png,Normal/images/Normal-3543.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3543.png,Normal/masks/Normal-3543.png,True,False,299,299,L,86.06,5779374db5a78b576163776c8dac6ff5
Normal-3544.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3544.png,Normal/images/Normal-3544.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3544.png,Normal/masks/Normal-3544.png,True,False,299,299,L,116.38,19d105f6287f60d432e260bab4403dd2
Normal-3545.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3545.png,Normal/images/Normal-3545.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3545.png,Normal/masks/Normal-3545.png,True,False,299,299,L,142.69,b44ec0c3712ae00bc2c033bce76b446c
Normal-3546.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3546.png,Normal/images/Normal-3546.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3546.png,Normal/masks/Normal-3546.png,True,False,299,299,L,113.74,9e2d7c78f1b035b5eb28e7866854cbe5
Normal-3547.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3547.png,Normal/images/Normal-3547.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3547.png,Normal/masks/Normal-3547.png,True,False,299,299,L,124.99,bc73692c778b662be18aad94a3855fb4
Normal-3548.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3548.png,Normal/images/Normal-3548.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3548.png,Normal/masks/Normal-3548.png,True,False,299,299,L,106.69,9106cdf534e69c95f675ba291bb3f963
Normal-3549.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3549.png,Normal/images/Normal-3549.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3549.png,Normal/masks/Normal-3549.png,True,False,299,299,L,150.26,218664e524a6c211bf31583b464df746
Normal-355.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-355.png,Normal/images/Normal-355.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-355.png,Normal/masks/Normal-355.png,True,False,299,299,L,130.12,31905d0bab3b868b4481c9835547b6ec
Normal-3550.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3550.png,Normal/images/Normal-3550.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3550.png,Normal/masks/Normal-3550.png,True,False,299,299,L,124.31,1cd6d01a13f890558b76ca8517285160
Normal-3551.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3551.png,Normal/images/Normal-3551.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3551.png,Normal/masks/Normal-3551.png,True,False,299,299,L,156.11,26a59e82f273aa37df5ff7f0b3f93394
Normal-3552.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3552.png,Normal/images/Normal-3552.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3552.png,Normal/masks/Normal-3552.png,True,False,299,299,L,126.08,135133b35b1cb80dab68021859a6b1ee
Normal-3553.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3553.png,Normal/images/Normal-3553.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3553.png,Normal/masks/Normal-3553.png,True,False,299,299,L,106.22,abbf492f27051f150690de4752384815
Normal-3554.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3554.png,Normal/images/Normal-3554.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3554.png,Normal/masks/Normal-3554.png,True,False,299,299,L,171.52,7548d78cfa2289072604a398568f2688
Normal-3555.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3555.png,Normal/images/Normal-3555.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3555.png,Normal/masks/Normal-3555.png,True,False,299,299,L,134.84,d92348db0aab58c000f44333ed15e953
Normal-3556.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3556.png,Normal/images/Normal-3556.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3556.png,Normal/masks/Normal-3556.png,True,False,299,299,L,153.73,307c9a4c950f4bb76ff012e985ebe85b
Normal-3557.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3557.png,Normal/images/Normal-3557.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3557.png,Normal/masks/Normal-3557.png,True,False,299,299,L,115.17,67ba3536d373f7cb8e158f14bfbc4c4c
Normal-3558.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3558.png,Normal/images/Normal-3558.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3558.png,Normal/masks/Normal-3558.png,True,False,299,299,L,109.38,fdd9e552c8b0638eeb28f2aeaa25b449
Normal-3559.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3559.png,Normal/images/Normal-3559.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3559.png,Normal/masks/Normal-3559.png,True,False,299,299,L,86.01,3ef6fb69470039ae713d975758a2a400
Normal-356.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-356.png,Normal/images/Normal-356.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-356.png,Normal/masks/Normal-356.png,True,False,299,299,L,137.39,8235c7a2f24c8d05bba371f3eb74e097
Normal-3560.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3560.png,Normal/images/Normal-3560.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3560.png,Normal/masks/Normal-3560.png,True,False,299,299,L,123.08,8cf3a9beada8430692da6566eebbd7d3
Normal-3561.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3561.png,Normal/images/Normal-3561.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3561.png,Normal/masks/Normal-3561.png,True,False,299,299,L,155.68,653913d98a0f17a1a130d04fa371796b
Normal-3562.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3562.png,Normal/images/Normal-3562.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3562.png,Normal/masks/Normal-3562.png,True,False,299,299,L,156.74,2b8e7f1b600f084d4250da0bbd6d5014
Normal-3563.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3563.png,Normal/images/Normal-3563.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3563.png,Normal/masks/Normal-3563.png,True,False,299,299,L,139.38,814041b4a4c89d16667fc9b13beb62ce
Normal-3564.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3564.png,Normal/images/Normal-3564.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3564.png,Normal/masks/Normal-3564.png,True,False,299,299,L,122.56,b1e611c28a32466415d4498df7739be5
Normal-3565.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3565.png,Normal/images/Normal-3565.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3565.png,Normal/masks/Normal-3565.png,True,False,299,299,L,132.96,683e67ccddd3e1f6d89d49cffa139b03
Normal-3566.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3566.png,Normal/images/Normal-3566.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3566.png,Normal/masks/Normal-3566.png,True,False,299,299,L,116.59,f2f4ff2dd6cc46e5065dce768e410ca0
Normal-3567.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3567.png,Normal/images/Normal-3567.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3567.png,Normal/masks/Normal-3567.png,True,False,299,299,L,119.24,83622531e5ab553b08b5c9e5b6cc095b
Normal-3568.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3568.png,Normal/images/Normal-3568.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3568.png,Normal/masks/Normal-3568.png,True,False,299,299,L,134.91,212c97c357e19fe91be5ceea4d5ae92b
Normal-3569.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3569.png,Normal/images/Normal-3569.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3569.png,Normal/masks/Normal-3569.png,True,False,299,299,L,118.84,bcd504b6dbbd1c7dc93e7c1ac64d116d
Normal-357.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-357.png,Normal/images/Normal-357.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-357.png,Normal/masks/Normal-357.png,True,False,299,299,L,93.49,e477d8ef6971ca4f2c9d578e8068d998
Normal-3570.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3570.png,Normal/images/Normal-3570.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3570.png,Normal/masks/Normal-3570.png,True,False,299,299,L,105.8,6eab07335b8994575c2c1df94d89f34d
Normal-3571.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3571.png,Normal/images/Normal-3571.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3571.png,Normal/masks/Normal-3571.png,True,False,299,299,L,105.87,ea0cfedb51732d5d70819d4a25b4d238
Normal-3572.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3572.png,Normal/images/Normal-3572.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3572.png,Normal/masks/Normal-3572.png,True,False,299,299,L,129.56,956482c0256006ff03514aa7a914da63
Normal-3573.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3573.png,Normal/images/Normal-3573.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3573.png,Normal/masks/Normal-3573.png,True,False,299,299,L,182.1,4c6588fac1bb93571b14c07fb2664ec3
Normal-3574.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3574.png,Normal/images/Normal-3574.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3574.png,Normal/masks/Normal-3574.png,True,False,299,299,L,125.95,965d68ad5a4389654fb2810731ce9f66
Normal-3575.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3575.png,Normal/images/Normal-3575.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3575.png,Normal/masks/Normal-3575.png,True,False,299,299,L,165.33,97d56c9dc6fae643b589f525f3485bb5
Normal-3576.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3576.png,Normal/images/Normal-3576.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3576.png,Normal/masks/Normal-3576.png,True,False,299,299,L,117.0,6b9ccd0f9838753572f16481c644f597
Normal-3577.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3577.png,Normal/images/Normal-3577.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3577.png,Normal/masks/Normal-3577.png,True,False,299,299,L,152.27,ec0ff4e968427a07c47806286729cf4a
Normal-3578.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3578.png,Normal/images/Normal-3578.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3578.png,Normal/masks/Normal-3578.png,True,False,299,299,L,95.03,fddc4e6c76b7501f1e2bc28740d09e40
Normal-3579.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3579.png,Normal/images/Normal-3579.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3579.png,Normal/masks/Normal-3579.png,True,False,299,299,L,134.87,b836f48c68421ee1c77b1ed30feaf685
Normal-358.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-358.png,Normal/images/Normal-358.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-358.png,Normal/masks/Normal-358.png,True,False,299,299,L,143.59,7dbf8d5d5822cba4da7b8221a8deaf6e
Normal-3580.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3580.png,Normal/images/Normal-3580.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3580.png,Normal/masks/Normal-3580.png,True,False,299,299,L,116.16,9da7775d495ba85df74417e3e7b79b72
Normal-3581.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3581.png,Normal/images/Normal-3581.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3581.png,Normal/masks/Normal-3581.png,True,False,299,299,L,125.53,8644a5497eb545f9f8f8b87f6c1e61db
Normal-3582.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3582.png,Normal/images/Normal-3582.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3582.png,Normal/masks/Normal-3582.png,True,False,299,299,L,163.75,7dace52284797bcf55d157208916f104
Normal-3583.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3583.png,Normal/images/Normal-3583.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3583.png,Normal/masks/Normal-3583.png,True,False,299,299,L,108.89,5d6e576a55ac926ae6b142c92b9af972
Normal-3584.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3584.png,Normal/images/Normal-3584.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3584.png,Normal/masks/Normal-3584.png,True,False,299,299,L,126.79,23b540f89d3a8bdc21ad45a0545c9501
Normal-3585.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3585.png,Normal/images/Normal-3585.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3585.png,Normal/masks/Normal-3585.png,True,False,299,299,L,96.53,59fd2eb25e61300da2132c582ca0bcbc
Normal-3586.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3586.png,Normal/images/Normal-3586.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3586.png,Normal/masks/Normal-3586.png,True,False,299,299,L,161.88,f158eae75766b617d0962fd0bb6a3272
Normal-3587.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3587.png,Normal/images/Normal-3587.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3587.png,Normal/masks/Normal-3587.png,True,False,299,299,L,98.37,1ecc44faf704200f318fc444720eb383
Normal-3588.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3588.png,Normal/images/Normal-3588.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3588.png,Normal/masks/Normal-3588.png,True,False,299,299,L,138.77,d75c3060107e6eb66c715db42ff55a5f
Normal-3589.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3589.png,Normal/images/Normal-3589.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3589.png,Normal/masks/Normal-3589.png,True,False,299,299,L,135.89,9e201b9d646da73dad610d8ac6ccfd58
Normal-359.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-359.png,Normal/images/Normal-359.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-359.png,Normal/masks/Normal-359.png,True,False,299,299,L,122.14,a42e4c3f3a69918aa6f30db2449ae57b
Normal-3590.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3590.png,Normal/images/Normal-3590.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3590.png,Normal/masks/Normal-3590.png,True,False,299,299,L,163.34,711a0b857ea6538e68f2af79a3d50732
Normal-3591.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3591.png,Normal/images/Normal-3591.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3591.png,Normal/masks/Normal-3591.png,True,False,299,299,L,98.57,dcebba4ddd6061177029adbbe618c211
Normal-3592.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3592.png,Normal/images/Normal-3592.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3592.png,Normal/masks/Normal-3592.png,True,False,299,299,L,118.14,d6d4f63cdc95e4fd118caf6fb5f35e32
Normal-3593.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3593.png,Normal/images/Normal-3593.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3593.png,Normal/masks/Normal-3593.png,True,False,299,299,L,103.27,b5f572d35750343a3ba3b068453a495a
Normal-3594.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3594.png,Normal/images/Normal-3594.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3594.png,Normal/masks/Normal-3594.png,True,False,299,299,L,127.89,c2a6d48b5e1f20d894b407843c87fa79
Normal-3595.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3595.png,Normal/images/Normal-3595.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3595.png,Normal/masks/Normal-3595.png,True,False,299,299,L,126.67,86e76fa076e78ab095a48bacd39a8a76
Normal-3596.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3596.png,Normal/images/Normal-3596.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3596.png,Normal/masks/Normal-3596.png,True,False,299,299,L,116.35,f2807b2655d5a2a2423e19af259b2bb0
Normal-3597.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3597.png,Normal/images/Normal-3597.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3597.png,Normal/masks/Normal-3597.png,True,False,299,299,L,155.21,f66134b0e2bbc14a53666691e3979d46
Normal-3598.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3598.png,Normal/images/Normal-3598.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3598.png,Normal/masks/Normal-3598.png,True,False,299,299,L,134.42,ec8f1bf6c67e86b85776f70db20f6173
Normal-3599.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3599.png,Normal/images/Normal-3599.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3599.png,Normal/masks/Normal-3599.png,True,False,299,299,L,114.31,f821b91c5a11ee46e779d8a596d3f252
Normal-36.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-36.png,Normal/images/Normal-36.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-36.png,Normal/masks/Normal-36.png,True,False,299,299,L,119.37,12c92ff19b1ec99a6d6d9ac49ead0e42
Normal-360.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-360.png,Normal/images/Normal-360.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-360.png,Normal/masks/Normal-360.png,True,False,299,299,L,150.19,a9ea5f721b38d7c76df5c6824521d9b9
Normal-3600.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3600.png,Normal/images/Normal-3600.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3600.png,Normal/masks/Normal-3600.png,True,False,299,299,L,129.03,d8233f6fde7981b54eca352ec66aec57
Normal-3601.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3601.png,Normal/images/Normal-3601.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3601.png,Normal/masks/Normal-3601.png,True,False,299,299,L,162.4,a7f4f641c34979aea346a064c9a662d5
Normal-3602.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3602.png,Normal/images/Normal-3602.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3602.png,Normal/masks/Normal-3602.png,True,False,299,299,L,128.61,f32a520403e8828ae800848ea0d1cb4f
Normal-3603.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3603.png,Normal/images/Normal-3603.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3603.png,Normal/masks/Normal-3603.png,True,False,299,299,L,158.56,6942f1edf48691bccebea2cde8f14d5d
Normal-3604.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3604.png,Normal/images/Normal-3604.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3604.png,Normal/masks/Normal-3604.png,True,False,299,299,L,140.82,3f020edae8f6e0e66280650ae7a97292
Normal-3605.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3605.png,Normal/images/Normal-3605.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3605.png,Normal/masks/Normal-3605.png,True,False,299,299,L,187.02,98a2bc7569de65424facb51a8419246a
Normal-3606.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3606.png,Normal/images/Normal-3606.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3606.png,Normal/masks/Normal-3606.png,True,False,299,299,L,140.9,21db12fdba9699ea037aceead619a82e
Normal-3607.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3607.png,Normal/images/Normal-3607.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3607.png,Normal/masks/Normal-3607.png,True,False,299,299,L,169.28,ac3f424ea72805c08b5dcc4b9b4bf5d0
Normal-3608.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3608.png,Normal/images/Normal-3608.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3608.png,Normal/masks/Normal-3608.png,True,False,299,299,L,127.02,4f4748b10160e29655cc38602f2ffd82
Normal-3609.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3609.png,Normal/images/Normal-3609.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3609.png,Normal/masks/Normal-3609.png,True,False,299,299,L,105.22,bed09ec4cd529dc0b7793d840d6aef5d
Normal-361.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-361.png,Normal/images/Normal-361.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-361.png,Normal/masks/Normal-361.png,True,False,299,299,L,131.22,aa0bc40efbb15510602193cbd17e479e
Normal-3610.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3610.png,Normal/images/Normal-3610.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3610.png,Normal/masks/Normal-3610.png,True,False,299,299,L,176.38,ece5c53d7773c93ce27e46d85917f6f0
Normal-3611.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3611.png,Normal/images/Normal-3611.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3611.png,Normal/masks/Normal-3611.png,True,False,299,299,L,121.46,098d2d819b29fe6e6e6c2e476bd213c3
Normal-3612.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3612.png,Normal/images/Normal-3612.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3612.png,Normal/masks/Normal-3612.png,True,False,299,299,L,109.59,25580b618d428fa300dd7213ca23039c
Normal-3613.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3613.png,Normal/images/Normal-3613.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3613.png,Normal/masks/Normal-3613.png,True,False,299,299,L,149.05,7e3b728c9263c9df384953f6ede064e6
Normal-3614.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3614.png,Normal/images/Normal-3614.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3614.png,Normal/masks/Normal-3614.png,True,False,299,299,L,111.79,86a2eabbedbb141e964bacf26560e225
Normal-3615.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3615.png,Normal/images/Normal-3615.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3615.png,Normal/masks/Normal-3615.png,True,False,299,299,L,114.53,be75af0f38746464cc5a262e14ad4039
Normal-3616.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3616.png,Normal/images/Normal-3616.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3616.png,Normal/masks/Normal-3616.png,True,False,299,299,L,101.87,214b07c0b8c02add703085a78f0ba49b
Normal-3617.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3617.png,Normal/images/Normal-3617.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3617.png,Normal/masks/Normal-3617.png,True,False,299,299,L,120.53,4ec07eed69c33528132a45921b8a8a22
Normal-3618.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3618.png,Normal/images/Normal-3618.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3618.png,Normal/masks/Normal-3618.png,True,False,299,299,L,152.52,b9f4e7855d132d128740f61089040834
Normal-3619.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3619.png,Normal/images/Normal-3619.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3619.png,Normal/masks/Normal-3619.png,True,False,299,299,L,163.91,92d6351f2b3eef9fe31ce8dc737add40
Normal-362.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-362.png,Normal/images/Normal-362.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-362.png,Normal/masks/Normal-362.png,True,False,299,299,L,131.06,f75ad155df2d585204e7de097f6067e9
Normal-3620.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3620.png,Normal/images/Normal-3620.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3620.png,Normal/masks/Normal-3620.png,True,False,299,299,L,170.96,b165a574c7812fb4c7e2cf27834642c1
Normal-3621.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3621.png,Normal/images/Normal-3621.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3621.png,Normal/masks/Normal-3621.png,True,False,299,299,L,97.62,74e7d7ec6fbace426839dd36fe9072ae
Normal-3622.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3622.png,Normal/images/Normal-3622.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3622.png,Normal/masks/Normal-3622.png,True,False,299,299,L,117.53,badfd8a66236d51a683704d637ab43c8
Normal-3623.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3623.png,Normal/images/Normal-3623.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3623.png,Normal/masks/Normal-3623.png,True,False,299,299,L,113.23,42cb5eefc27ae33d054f81f11a2bc7ce
Normal-3624.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3624.png,Normal/images/Normal-3624.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3624.png,Normal/masks/Normal-3624.png,True,False,299,299,L,110.8,c50ce81d2d6d47b33bf44bb344074c6c
Normal-3625.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3625.png,Normal/images/Normal-3625.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3625.png,Normal/masks/Normal-3625.png,True,False,299,299,L,109.48,811cdee6b1f08fbf57d7bdb01799404e
Normal-3626.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3626.png,Normal/images/Normal-3626.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3626.png,Normal/masks/Normal-3626.png,True,False,299,299,L,164.59,8a762c64405ece808158b95107ea2c4d
Normal-3627.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3627.png,Normal/images/Normal-3627.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3627.png,Normal/masks/Normal-3627.png,True,False,299,299,L,128.29,95a13258f26596362444d9427e8c2173
Normal-3628.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3628.png,Normal/images/Normal-3628.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3628.png,Normal/masks/Normal-3628.png,True,False,299,299,L,137.18,e921413fe30f6dfb141c1d2491b1e369
Normal-3629.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3629.png,Normal/images/Normal-3629.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3629.png,Normal/masks/Normal-3629.png,True,False,299,299,L,102.38,374fb4f179fd560e16e47f3a57002352
Normal-363.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-363.png,Normal/images/Normal-363.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-363.png,Normal/masks/Normal-363.png,True,False,299,299,L,118.59,c04be080ae2bb17699833602c6c59bc6
Normal-3630.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3630.png,Normal/images/Normal-3630.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3630.png,Normal/masks/Normal-3630.png,True,False,299,299,L,145.67,61893ace62a7796cc144526dac00ea86
Normal-3631.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3631.png,Normal/images/Normal-3631.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3631.png,Normal/masks/Normal-3631.png,True,False,299,299,L,132.11,d6ca8ac15d1adc0e6f533e42223ce37e
Normal-3632.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3632.png,Normal/images/Normal-3632.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3632.png,Normal/masks/Normal-3632.png,True,False,299,299,L,85.83,52f1dad8e4e5946e8c60263f39dfee27
Normal-3633.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3633.png,Normal/images/Normal-3633.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3633.png,Normal/masks/Normal-3633.png,True,False,299,299,L,166.28,824669d3265409eecc2299b389cf8422
Normal-3634.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3634.png,Normal/images/Normal-3634.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3634.png,Normal/masks/Normal-3634.png,True,False,299,299,L,135.97,c66f8e38afeb6ca62fb86c672c7621f6
Normal-3635.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3635.png,Normal/images/Normal-3635.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3635.png,Normal/masks/Normal-3635.png,True,False,299,299,L,127.51,cc010d53b4506c410198eff6258f8b40
Normal-3636.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3636.png,Normal/images/Normal-3636.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3636.png,Normal/masks/Normal-3636.png,True,False,299,299,L,126.54,8066c18bb583b0a293bc1612819d0044
Normal-3637.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3637.png,Normal/images/Normal-3637.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3637.png,Normal/masks/Normal-3637.png,True,False,299,299,L,95.9,3c132e74c44f0a9a1b6ee7bcaf699cf2
Normal-3638.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3638.png,Normal/images/Normal-3638.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3638.png,Normal/masks/Normal-3638.png,True,False,299,299,L,166.74,a37186c9f9a3fa36a14ce52e0412c0ee
Normal-3639.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3639.png,Normal/images/Normal-3639.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3639.png,Normal/masks/Normal-3639.png,True,False,299,299,L,148.41,439b36ee88ec01f76b2ace0cf83622a8
Normal-364.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-364.png,Normal/images/Normal-364.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-364.png,Normal/masks/Normal-364.png,True,False,299,299,L,132.51,f8f460085a86e02234feb46de8f0139e
Normal-3640.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3640.png,Normal/images/Normal-3640.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3640.png,Normal/masks/Normal-3640.png,True,False,299,299,L,121.64,28cb2bc9d9a2be8d72796c242fde920e
Normal-3641.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3641.png,Normal/images/Normal-3641.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3641.png,Normal/masks/Normal-3641.png,True,False,299,299,L,105.5,9518074a7e3c6bcef413e5a74754eaa1
Normal-3642.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3642.png,Normal/images/Normal-3642.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3642.png,Normal/masks/Normal-3642.png,True,False,299,299,L,124.46,536726113e3e1ffc6f62beb4aa0b82d3
Normal-3643.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3643.png,Normal/images/Normal-3643.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3643.png,Normal/masks/Normal-3643.png,True,False,299,299,L,130.61,92f06843e9fb3866436abc4382f234f8
Normal-3644.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3644.png,Normal/images/Normal-3644.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3644.png,Normal/masks/Normal-3644.png,True,False,299,299,L,112.06,a943e9c41d563df6fcba40df04faa176
Normal-3645.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3645.png,Normal/images/Normal-3645.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3645.png,Normal/masks/Normal-3645.png,True,False,299,299,L,123.28,823676b0c5b88cb29dce4efac5452eeb
Normal-3646.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3646.png,Normal/images/Normal-3646.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3646.png,Normal/masks/Normal-3646.png,True,False,299,299,L,145.0,c853a3d632b76eaeaa1051e8dd4eeb96
Normal-3647.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3647.png,Normal/images/Normal-3647.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3647.png,Normal/masks/Normal-3647.png,True,False,299,299,L,108.73,996a814479ee3587a4f1d2ca386edb6d
Normal-3648.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3648.png,Normal/images/Normal-3648.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3648.png,Normal/masks/Normal-3648.png,True,False,299,299,L,109.42,e6dcb0444d040ec38360bca89ed790b1
Normal-3649.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3649.png,Normal/images/Normal-3649.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3649.png,Normal/masks/Normal-3649.png,True,False,299,299,L,126.6,8e65b461e4bfa7308c2f22494e3a2277
Normal-365.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-365.png,Normal/images/Normal-365.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-365.png,Normal/masks/Normal-365.png,True,False,299,299,L,117.28,d95279010611b63182d71a7efc359ff8
Normal-3650.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3650.png,Normal/images/Normal-3650.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3650.png,Normal/masks/Normal-3650.png,True,False,299,299,L,101.24,d7970e1920e7091aa5c6a4286fe6dd6e
Normal-3651.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3651.png,Normal/images/Normal-3651.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3651.png,Normal/masks/Normal-3651.png,True,False,299,299,L,154.73,be182661770f0598ffe0e8dd34af5e17
Normal-3652.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3652.png,Normal/images/Normal-3652.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3652.png,Normal/masks/Normal-3652.png,True,False,299,299,L,146.47,be77e11803609c9df2393a752204d4e2
Normal-3653.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3653.png,Normal/images/Normal-3653.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3653.png,Normal/masks/Normal-3653.png,True,False,299,299,L,95.79,1d8b9e6d9442f1f06db954b50f7e24e5
Normal-3654.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3654.png,Normal/images/Normal-3654.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3654.png,Normal/masks/Normal-3654.png,True,False,299,299,L,152.03,f18a563d5f41f2ccff1944cb9100bacb
Normal-3655.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3655.png,Normal/images/Normal-3655.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3655.png,Normal/masks/Normal-3655.png,True,False,299,299,L,126.07,533e46647cb1cc9e9acffff97bb3f861
Normal-3656.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3656.png,Normal/images/Normal-3656.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3656.png,Normal/masks/Normal-3656.png,True,False,299,299,L,176.57,2e71617610b4b0fbb8eef132c1b8ee40
Normal-3657.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3657.png,Normal/images/Normal-3657.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3657.png,Normal/masks/Normal-3657.png,True,False,299,299,L,143.84,7c24d620b77b3226656bf5dae8904275
Normal-3658.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3658.png,Normal/images/Normal-3658.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3658.png,Normal/masks/Normal-3658.png,True,False,299,299,L,97.35,045cc9fd115197e145817cba22322118
Normal-3659.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3659.png,Normal/images/Normal-3659.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3659.png,Normal/masks/Normal-3659.png,True,False,299,299,L,116.45,af9579b8c381d86ba73f70b002fc4af6
Normal-366.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-366.png,Normal/images/Normal-366.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-366.png,Normal/masks/Normal-366.png,True,False,299,299,L,126.46,acf60475c5d1db84655b7e9b866ae3c4
Normal-3660.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3660.png,Normal/images/Normal-3660.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3660.png,Normal/masks/Normal-3660.png,True,False,299,299,L,139.53,de245d687a50580967f66eff8ae854d0
Normal-3661.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3661.png,Normal/images/Normal-3661.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3661.png,Normal/masks/Normal-3661.png,True,False,299,299,L,104.07,24676ea0652dcb5a6057d0959c64aa5e
Normal-3662.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3662.png,Normal/images/Normal-3662.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3662.png,Normal/masks/Normal-3662.png,True,False,299,299,L,120.25,db3bbf8b1d3ad3a8c91a8ee1d008b289
Normal-3663.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3663.png,Normal/images/Normal-3663.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3663.png,Normal/masks/Normal-3663.png,True,False,299,299,L,178.15,849749e804676465a2a892057b843789
Normal-3664.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3664.png,Normal/images/Normal-3664.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3664.png,Normal/masks/Normal-3664.png,True,False,299,299,L,183.28,99df706e9873b8bffe22eaba8ffdbea0
Normal-3665.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3665.png,Normal/images/Normal-3665.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3665.png,Normal/masks/Normal-3665.png,True,False,299,299,L,116.54,45915a184661a8a0ea8e677e3ccc700c
Normal-3666.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3666.png,Normal/images/Normal-3666.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3666.png,Normal/masks/Normal-3666.png,True,False,299,299,L,163.7,4e2ffc5add9a0025d61affb6ca46a426
Normal-3667.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3667.png,Normal/images/Normal-3667.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3667.png,Normal/masks/Normal-3667.png,True,False,299,299,L,129.81,90fcc720343c5b6d36b4d9c7bfea3db4
Normal-3668.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3668.png,Normal/images/Normal-3668.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3668.png,Normal/masks/Normal-3668.png,True,False,299,299,L,129.86,4991862e65aaba296a5bcd91cfbd56d1
Normal-3669.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3669.png,Normal/images/Normal-3669.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3669.png,Normal/masks/Normal-3669.png,True,False,299,299,L,114.24,359897434786a73ef4911d5b95b22133
Normal-367.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-367.png,Normal/images/Normal-367.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-367.png,Normal/masks/Normal-367.png,True,False,299,299,L,127.37,040951cd1f5f8e5ef1aed33056428c7a
Normal-3670.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3670.png,Normal/images/Normal-3670.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3670.png,Normal/masks/Normal-3670.png,True,False,299,299,L,117.23,0f87d9bfd4b597201dd641ceb81f450b
Normal-3671.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3671.png,Normal/images/Normal-3671.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3671.png,Normal/masks/Normal-3671.png,True,False,299,299,L,166.04,2891fb1b766057682dc82282912d0591
Normal-3672.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3672.png,Normal/images/Normal-3672.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3672.png,Normal/masks/Normal-3672.png,True,False,299,299,L,126.79,8145bd812313c955bcf6e9cf0756a04b
Normal-3673.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3673.png,Normal/images/Normal-3673.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3673.png,Normal/masks/Normal-3673.png,True,False,299,299,L,156.83,c2b3da9291bc1c38b6c4c17eda0e8caa
Normal-3674.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3674.png,Normal/images/Normal-3674.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3674.png,Normal/masks/Normal-3674.png,True,False,299,299,L,100.3,adad32e05bb96cfcf334b4fd6ce60d8e
Normal-3675.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3675.png,Normal/images/Normal-3675.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3675.png,Normal/masks/Normal-3675.png,True,False,299,299,L,107.09,d7c0fd0cbce3294bad31153ca446feee
Normal-3676.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3676.png,Normal/images/Normal-3676.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3676.png,Normal/masks/Normal-3676.png,True,False,299,299,L,144.44,1ca0df00480c2f24f826eb64a40a78b8
Normal-3677.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3677.png,Normal/images/Normal-3677.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3677.png,Normal/masks/Normal-3677.png,True,False,299,299,L,122.95,f7fecfbca71a192459db1d3696367734
Normal-3678.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3678.png,Normal/images/Normal-3678.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3678.png,Normal/masks/Normal-3678.png,True,False,299,299,L,122.26,e65c6b0cc8bf484b247f8711d77e9462
Normal-3679.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3679.png,Normal/images/Normal-3679.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3679.png,Normal/masks/Normal-3679.png,True,False,299,299,L,160.4,9d6ef75177fc50bf34129a442a4cfd2d
Normal-368.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-368.png,Normal/images/Normal-368.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-368.png,Normal/masks/Normal-368.png,True,False,299,299,L,122.51,18a947d05b5692c9f81db8e742fa64c8
Normal-3680.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3680.png,Normal/images/Normal-3680.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3680.png,Normal/masks/Normal-3680.png,True,False,299,299,L,174.71,7cc7920e2c01212a7d536de849dbcd41
Normal-3681.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3681.png,Normal/images/Normal-3681.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3681.png,Normal/masks/Normal-3681.png,True,False,299,299,L,91.8,1c094d7c1932241933d535c5162933ff
Normal-3682.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3682.png,Normal/images/Normal-3682.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3682.png,Normal/masks/Normal-3682.png,True,False,299,299,L,114.04,e142b1b658ec7bbc20ce29d4f81594e3
Normal-3683.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3683.png,Normal/images/Normal-3683.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3683.png,Normal/masks/Normal-3683.png,True,False,299,299,L,128.07,aa55a89bbc006bfd92c031cf60a993e5
Normal-3684.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3684.png,Normal/images/Normal-3684.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3684.png,Normal/masks/Normal-3684.png,True,False,299,299,L,141.98,76214f5942574428ba7ce2b8ad1e218b
Normal-3685.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3685.png,Normal/images/Normal-3685.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3685.png,Normal/masks/Normal-3685.png,True,False,299,299,L,133.12,f2aa4abcb1ee8a36f0dba489ffa00755
Normal-3686.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3686.png,Normal/images/Normal-3686.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3686.png,Normal/masks/Normal-3686.png,True,False,299,299,L,139.33,98a63ffe248233526dca74c465765c29
Normal-3687.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3687.png,Normal/images/Normal-3687.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3687.png,Normal/masks/Normal-3687.png,True,False,299,299,L,107.7,8edabdb5f428a6983b9392a5cf6b8c38
Normal-3688.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3688.png,Normal/images/Normal-3688.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3688.png,Normal/masks/Normal-3688.png,True,False,299,299,L,153.08,5376be49ff5d6478cdf05242b977ed15
Normal-3689.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3689.png,Normal/images/Normal-3689.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3689.png,Normal/masks/Normal-3689.png,True,False,299,299,L,137.49,41280b2ed136462a83923b14339d1279
Normal-369.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-369.png,Normal/images/Normal-369.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-369.png,Normal/masks/Normal-369.png,True,False,299,299,L,123.48,4e0d2c084b7642eaaf0aeab2730f3149
Normal-3690.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3690.png,Normal/images/Normal-3690.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3690.png,Normal/masks/Normal-3690.png,True,False,299,299,L,155.69,7ea0263e494bbbd9ce774454de6b8434
Normal-3691.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3691.png,Normal/images/Normal-3691.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3691.png,Normal/masks/Normal-3691.png,True,False,299,299,L,106.21,82942010cc39da29dc015da0d0985bf1
Normal-3692.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3692.png,Normal/images/Normal-3692.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3692.png,Normal/masks/Normal-3692.png,True,False,299,299,L,106.7,febadd440f6f61860f696afbe4971a68
Normal-3693.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3693.png,Normal/images/Normal-3693.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3693.png,Normal/masks/Normal-3693.png,True,False,299,299,L,116.78,bb52b5b7b6b9f6cb0778b09e99ea4dbc
Normal-3694.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3694.png,Normal/images/Normal-3694.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3694.png,Normal/masks/Normal-3694.png,True,False,299,299,L,135.15,ba3da9e45c4c6e41a920027cea2bb2c5
Normal-3695.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3695.png,Normal/images/Normal-3695.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3695.png,Normal/masks/Normal-3695.png,True,False,299,299,L,110.99,6e1131cc17b0d567d48c892a96f4b06a
Normal-3696.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3696.png,Normal/images/Normal-3696.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3696.png,Normal/masks/Normal-3696.png,True,False,299,299,L,125.13,bdcd29ddddf7997d88ef6fddc5638f2c
Normal-3697.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3697.png,Normal/images/Normal-3697.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3697.png,Normal/masks/Normal-3697.png,True,False,299,299,L,104.28,7a4a6d310edc3760517cf484856a90be
Normal-3698.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3698.png,Normal/images/Normal-3698.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3698.png,Normal/masks/Normal-3698.png,True,False,299,299,L,144.45,9d3095171447dfc2ac05ca603706d4f9
Normal-3699.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3699.png,Normal/images/Normal-3699.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3699.png,Normal/masks/Normal-3699.png,True,False,299,299,L,129.74,c0b631e6835174ed548b1280486f3066
Normal-37.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-37.png,Normal/images/Normal-37.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-37.png,Normal/masks/Normal-37.png,True,False,299,299,L,111.06,c1907d4d17c3c049d287ae4e0c5ac825
Normal-370.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-370.png,Normal/images/Normal-370.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-370.png,Normal/masks/Normal-370.png,True,False,299,299,L,100.35,290f04d3c53d01077becf0ee5559c24f
Normal-3700.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3700.png,Normal/images/Normal-3700.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3700.png,Normal/masks/Normal-3700.png,True,False,299,299,L,135.98,03fd439611b6aec026046e4021cee00e
Normal-3701.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3701.png,Normal/images/Normal-3701.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3701.png,Normal/masks/Normal-3701.png,True,False,299,299,L,110.71,847a10b475807db8856238b555901e31
Normal-3702.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3702.png,Normal/images/Normal-3702.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3702.png,Normal/masks/Normal-3702.png,True,False,299,299,L,137.22,043dfdbdeb527c5d9b8038084bd00ca5
Normal-3703.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3703.png,Normal/images/Normal-3703.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3703.png,Normal/masks/Normal-3703.png,True,False,299,299,L,122.39,da593cc88340a73a62c11e1fa8557dda
Normal-3704.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3704.png,Normal/images/Normal-3704.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3704.png,Normal/masks/Normal-3704.png,True,False,299,299,L,121.43,5b0be60b3a44bc32f0a78a5244c70849
Normal-3705.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3705.png,Normal/images/Normal-3705.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3705.png,Normal/masks/Normal-3705.png,True,False,299,299,L,168.99,4268371897e3fee6faad0d7f80f4463a
Normal-3706.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3706.png,Normal/images/Normal-3706.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3706.png,Normal/masks/Normal-3706.png,True,False,299,299,L,115.03,12338b56a9b862c4a3e32696e2711e4e
Normal-3707.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3707.png,Normal/images/Normal-3707.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3707.png,Normal/masks/Normal-3707.png,True,False,299,299,L,120.12,964b3fc7f4b3f66f56db4f7cd3868e1a
Normal-3708.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3708.png,Normal/images/Normal-3708.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3708.png,Normal/masks/Normal-3708.png,True,False,299,299,L,112.92,57db18ca8f4efb2ca7690c7afb58710d
Normal-3709.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3709.png,Normal/images/Normal-3709.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3709.png,Normal/masks/Normal-3709.png,True,False,299,299,L,105.41,153646b946e7f86ea8a6ef19c87ad6ad
Normal-371.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-371.png,Normal/images/Normal-371.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-371.png,Normal/masks/Normal-371.png,True,False,299,299,L,159.6,8ea197eb3f14ac12ba39de8d2972d468
Normal-3710.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3710.png,Normal/images/Normal-3710.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3710.png,Normal/masks/Normal-3710.png,True,False,299,299,L,102.75,6b7d0b44e6e4f30ff20b1a46be5a693d
Normal-3711.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3711.png,Normal/images/Normal-3711.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3711.png,Normal/masks/Normal-3711.png,True,False,299,299,L,141.2,45ec8ad69373de28727f029312266c68
Normal-3712.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3712.png,Normal/images/Normal-3712.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3712.png,Normal/masks/Normal-3712.png,True,False,299,299,L,124.04,bd117a20a88094eb264a16a91a036f15
Normal-3713.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3713.png,Normal/images/Normal-3713.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3713.png,Normal/masks/Normal-3713.png,True,False,299,299,L,143.91,76bd1bbb3585ae8087c3336b723024ed
Normal-3714.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3714.png,Normal/images/Normal-3714.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3714.png,Normal/masks/Normal-3714.png,True,False,299,299,L,115.05,0e683af091cb21b3064ac8846e124149
Normal-3715.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3715.png,Normal/images/Normal-3715.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3715.png,Normal/masks/Normal-3715.png,True,False,299,299,L,124.4,4d3d07eebbe94d37693b344217f42867
Normal-3716.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3716.png,Normal/images/Normal-3716.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3716.png,Normal/masks/Normal-3716.png,True,False,299,299,L,128.81,843d98e70ece05d4f0be77e3e8d6cfb3
Normal-3717.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3717.png,Normal/images/Normal-3717.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3717.png,Normal/masks/Normal-3717.png,True,False,299,299,L,152.45,753e47bca39e8dd59043d8eba13199c6
Normal-3718.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3718.png,Normal/images/Normal-3718.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3718.png,Normal/masks/Normal-3718.png,True,False,299,299,L,108.86,1e5a99014927cbb52aebe7ce5e11e069
Normal-3719.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3719.png,Normal/images/Normal-3719.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3719.png,Normal/masks/Normal-3719.png,True,False,299,299,L,139.83,5ae2f4aa2dfca24b2ee921dc1d32c311
Normal-372.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-372.png,Normal/images/Normal-372.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-372.png,Normal/masks/Normal-372.png,True,False,299,299,L,122.84,b696bf14096b1a29856704bd77463ce4
Normal-3720.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3720.png,Normal/images/Normal-3720.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3720.png,Normal/masks/Normal-3720.png,True,False,299,299,L,121.26,4aa471524a1853ec5cde60bf2f1d041e
Normal-3721.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3721.png,Normal/images/Normal-3721.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3721.png,Normal/masks/Normal-3721.png,True,False,299,299,L,117.95,b29bc08a45d6f81eec94904db236feba
Normal-3722.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3722.png,Normal/images/Normal-3722.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3722.png,Normal/masks/Normal-3722.png,True,False,299,299,L,167.06,db6dea8a9c7c138d0962889d06379664
Normal-3723.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3723.png,Normal/images/Normal-3723.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3723.png,Normal/masks/Normal-3723.png,True,False,299,299,L,108.62,fe57130a83161e3d86f28c261c388814
Normal-3724.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3724.png,Normal/images/Normal-3724.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3724.png,Normal/masks/Normal-3724.png,True,False,299,299,L,147.48,379efb51141ccc47e2ac5c3f83f62f21
Normal-3725.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3725.png,Normal/images/Normal-3725.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3725.png,Normal/masks/Normal-3725.png,True,False,299,299,L,122.94,58fab4935cd5ed6d3a7973b7e5404f16
Normal-3726.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3726.png,Normal/images/Normal-3726.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3726.png,Normal/masks/Normal-3726.png,True,False,299,299,L,115.45,ba7f93c0531925304e1c24601af6a6ea
Normal-3727.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3727.png,Normal/images/Normal-3727.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3727.png,Normal/masks/Normal-3727.png,True,False,299,299,L,156.05,a56b718c84806b18f9735483a3715fa0
Normal-3728.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3728.png,Normal/images/Normal-3728.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3728.png,Normal/masks/Normal-3728.png,True,False,299,299,L,115.33,7d38114e93b5db7abd2f659b249ca6b9
Normal-3729.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3729.png,Normal/images/Normal-3729.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3729.png,Normal/masks/Normal-3729.png,True,False,299,299,L,137.91,3d5e9858d70928e2031d0f2d97f3539f
Normal-373.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-373.png,Normal/images/Normal-373.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-373.png,Normal/masks/Normal-373.png,True,False,299,299,L,110.99,2610e085dafa260a1b8822c0ad53f9f6
Normal-3730.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3730.png,Normal/images/Normal-3730.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3730.png,Normal/masks/Normal-3730.png,True,False,299,299,L,135.22,1723d1f15ce6ea23577fd38416d6ac78
Normal-3731.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3731.png,Normal/images/Normal-3731.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3731.png,Normal/masks/Normal-3731.png,True,False,299,299,L,133.71,932082a9ccb38d28e24f0a67817aca23
Normal-3732.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3732.png,Normal/images/Normal-3732.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3732.png,Normal/masks/Normal-3732.png,True,False,299,299,L,106.65,d271aa8e9e000229df540eb2a681444e
Normal-3733.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3733.png,Normal/images/Normal-3733.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3733.png,Normal/masks/Normal-3733.png,True,False,299,299,L,163.63,a856af52eb4dd7f04a5656859c62620b
Normal-3734.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3734.png,Normal/images/Normal-3734.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3734.png,Normal/masks/Normal-3734.png,True,False,299,299,L,119.69,e81707cb9e248e39d062c69354a7eef7
Normal-3735.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3735.png,Normal/images/Normal-3735.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3735.png,Normal/masks/Normal-3735.png,True,False,299,299,L,124.51,2a1eccf4a69849cce728cd10b56f5594
Normal-3736.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3736.png,Normal/images/Normal-3736.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3736.png,Normal/masks/Normal-3736.png,True,False,299,299,L,166.41,c2262e9f6f8850f6c76a724e4d2adf0a
Normal-3737.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3737.png,Normal/images/Normal-3737.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3737.png,Normal/masks/Normal-3737.png,True,False,299,299,L,129.82,77a67876d25ce1578602fe0ae87af5e3
Normal-3738.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3738.png,Normal/images/Normal-3738.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3738.png,Normal/masks/Normal-3738.png,True,False,299,299,L,114.83,d33a5175f6e4e58fe9d648604bfd1bec
Normal-3739.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3739.png,Normal/images/Normal-3739.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3739.png,Normal/masks/Normal-3739.png,True,False,299,299,L,121.76,a6bbde6df7e0f9b21bd9f2b70677cb91
Normal-374.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-374.png,Normal/images/Normal-374.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-374.png,Normal/masks/Normal-374.png,True,False,299,299,L,132.42,e60f988650e033ec9ae339aa2874c722
Normal-3740.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3740.png,Normal/images/Normal-3740.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3740.png,Normal/masks/Normal-3740.png,True,False,299,299,L,130.06,0ffbd33cf1c946274cf95ea420e7ef58
Normal-3741.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3741.png,Normal/images/Normal-3741.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3741.png,Normal/masks/Normal-3741.png,True,False,299,299,L,112.06,d3cd075cc87b1c7585d9c94a4de7565a
Normal-3742.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3742.png,Normal/images/Normal-3742.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3742.png,Normal/masks/Normal-3742.png,True,False,299,299,L,137.57,4ccedd1b0bab8869455f574680267a79
Normal-3743.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3743.png,Normal/images/Normal-3743.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3743.png,Normal/masks/Normal-3743.png,True,False,299,299,L,149.24,a6d2fa47d2c54f140703ebc998290d5f
Normal-3744.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3744.png,Normal/images/Normal-3744.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3744.png,Normal/masks/Normal-3744.png,True,False,299,299,L,145.99,76707f79d0988cbb4445e762e59aa7c6
Normal-3745.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3745.png,Normal/images/Normal-3745.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3745.png,Normal/masks/Normal-3745.png,True,False,299,299,L,136.72,149ed6f80db8ff40f8778f29d4db85a3
Normal-3746.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3746.png,Normal/images/Normal-3746.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3746.png,Normal/masks/Normal-3746.png,True,False,299,299,L,119.59,51036feb6f2aca24e55e34d7f95ea120
Normal-3747.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3747.png,Normal/images/Normal-3747.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3747.png,Normal/masks/Normal-3747.png,True,False,299,299,L,114.49,99a0fb7e90ab059648474735879ac4bc
Normal-3748.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3748.png,Normal/images/Normal-3748.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3748.png,Normal/masks/Normal-3748.png,True,False,299,299,L,85.42,151cdd3f4b400d3e74096190034794bf
Normal-3749.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3749.png,Normal/images/Normal-3749.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3749.png,Normal/masks/Normal-3749.png,True,False,299,299,L,168.32,16d9ac104026204f694602a762d818e0
Normal-375.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-375.png,Normal/images/Normal-375.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-375.png,Normal/masks/Normal-375.png,True,False,299,299,L,132.36,90c12155ae9fce0bd903db49b23d32fc
Normal-3750.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3750.png,Normal/images/Normal-3750.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3750.png,Normal/masks/Normal-3750.png,True,False,299,299,L,164.47,c0fc4a4a46178196c329ee7c74e90579
Normal-3751.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3751.png,Normal/images/Normal-3751.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3751.png,Normal/masks/Normal-3751.png,True,False,299,299,L,171.54,0a07cba769467e1bdd3c3cf6cafe6e50
Normal-3752.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3752.png,Normal/images/Normal-3752.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3752.png,Normal/masks/Normal-3752.png,True,False,299,299,L,107.92,2fde2bc6d7f0ed0aa8c65a49a72b8ec2
Normal-3753.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3753.png,Normal/images/Normal-3753.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3753.png,Normal/masks/Normal-3753.png,True,False,299,299,L,175.41,7cef8d5634612e8c24293bf217ca94bf
Normal-3754.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3754.png,Normal/images/Normal-3754.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3754.png,Normal/masks/Normal-3754.png,True,False,299,299,L,77.52,87b2bfcf39502fe9869d7d9ed5c8d8db
Normal-3755.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3755.png,Normal/images/Normal-3755.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3755.png,Normal/masks/Normal-3755.png,True,False,299,299,L,134.1,8b1044a55a88aaedb10d8ea61b4b1e9d
Normal-3756.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3756.png,Normal/images/Normal-3756.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3756.png,Normal/masks/Normal-3756.png,True,False,299,299,L,126.73,92242db28d20a527c48256daed946e41
Normal-3757.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3757.png,Normal/images/Normal-3757.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3757.png,Normal/masks/Normal-3757.png,True,False,299,299,L,178.14,aa552405f84d9cfa7618bc768dde8363
Normal-3758.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3758.png,Normal/images/Normal-3758.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3758.png,Normal/masks/Normal-3758.png,True,False,299,299,L,110.12,0ab55db62d3afdbb3e53d257828c45bb
Normal-3759.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3759.png,Normal/images/Normal-3759.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3759.png,Normal/masks/Normal-3759.png,True,False,299,299,L,132.06,29206913b9a7d82a3429961dfac3b9f0
Normal-376.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-376.png,Normal/images/Normal-376.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-376.png,Normal/masks/Normal-376.png,True,False,299,299,L,103.99,137e1c011658541d19bb1208b290687a
Normal-3760.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3760.png,Normal/images/Normal-3760.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3760.png,Normal/masks/Normal-3760.png,True,False,299,299,L,125.95,3bfcef9ccb3f90e2fe5c1fc17d1f51df
Normal-3761.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3761.png,Normal/images/Normal-3761.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3761.png,Normal/masks/Normal-3761.png,True,False,299,299,L,96.64,d31934197ea8164580a91e7ece4901e9
Normal-3762.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3762.png,Normal/images/Normal-3762.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3762.png,Normal/masks/Normal-3762.png,True,False,299,299,L,72.41,347a5a74723e5f11398f126425343ee5
Normal-3763.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3763.png,Normal/images/Normal-3763.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3763.png,Normal/masks/Normal-3763.png,True,False,299,299,L,111.82,6f3502039e472fd2e700edab62bd9231
Normal-3764.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3764.png,Normal/images/Normal-3764.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3764.png,Normal/masks/Normal-3764.png,True,False,299,299,L,158.89,8873c0d85de872b425d4fb00ddfd0899
Normal-3765.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3765.png,Normal/images/Normal-3765.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3765.png,Normal/masks/Normal-3765.png,True,False,299,299,L,131.74,57b09da7b00ceac723a49e3b16ee5524
Normal-3766.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3766.png,Normal/images/Normal-3766.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3766.png,Normal/masks/Normal-3766.png,True,False,299,299,L,111.28,63bba90153ff210837ffb88b1fee87ce
Normal-3767.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3767.png,Normal/images/Normal-3767.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3767.png,Normal/masks/Normal-3767.png,True,False,299,299,L,125.98,03661e2d1db06caca1fcc5ea61906e97
Normal-3768.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3768.png,Normal/images/Normal-3768.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3768.png,Normal/masks/Normal-3768.png,True,False,299,299,L,122.85,6d593e2d75f9036fd31a950e137f6848
Normal-3769.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3769.png,Normal/images/Normal-3769.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3769.png,Normal/masks/Normal-3769.png,True,False,299,299,L,167.24,6908bc3f79fbff3427361e4fd5294c8a
Normal-377.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-377.png,Normal/images/Normal-377.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-377.png,Normal/masks/Normal-377.png,True,False,299,299,L,122.14,07c282d1985746b7ced303485c19f4b7
Normal-3770.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3770.png,Normal/images/Normal-3770.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3770.png,Normal/masks/Normal-3770.png,True,False,299,299,L,110.18,a2bc9f38406c80582cff816052a4ebc7
Normal-3771.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3771.png,Normal/images/Normal-3771.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3771.png,Normal/masks/Normal-3771.png,True,False,299,299,L,118.32,0f7cf92b33db9463ebf50504571ac1fe
Normal-3772.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3772.png,Normal/images/Normal-3772.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3772.png,Normal/masks/Normal-3772.png,True,False,299,299,L,174.13,f4fba9ec69bbf36631560c055186f9b7
Normal-3773.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3773.png,Normal/images/Normal-3773.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3773.png,Normal/masks/Normal-3773.png,True,False,299,299,L,143.16,e86bf92a614f1d2928606e86496c55c3
Normal-3774.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3774.png,Normal/images/Normal-3774.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3774.png,Normal/masks/Normal-3774.png,True,False,299,299,L,117.79,4e975b4cd1e7d8e4e57f68913b9f9f8b
Normal-3775.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3775.png,Normal/images/Normal-3775.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3775.png,Normal/masks/Normal-3775.png,True,False,299,299,L,165.15,5a60aa165416716719f9282a750ca57b
Normal-3776.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3776.png,Normal/images/Normal-3776.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3776.png,Normal/masks/Normal-3776.png,True,False,299,299,L,94.56,5a6a0b9ac48c007861a683c0e426ee9b
Normal-3777.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3777.png,Normal/images/Normal-3777.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3777.png,Normal/masks/Normal-3777.png,True,False,299,299,L,140.65,d6c02c632b017439bbdbe5a6decce8b7
Normal-3778.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3778.png,Normal/images/Normal-3778.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3778.png,Normal/masks/Normal-3778.png,True,False,299,299,L,128.66,cc05a5222b4c8c4c57cfb59bf94d93dd
Normal-3779.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3779.png,Normal/images/Normal-3779.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3779.png,Normal/masks/Normal-3779.png,True,False,299,299,L,121.46,a615e192abebec5b9afe7f7c4f8b9f1f
Normal-378.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-378.png,Normal/images/Normal-378.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-378.png,Normal/masks/Normal-378.png,True,False,299,299,L,112.67,df6c31931f75c955fc0301759ede9e70
Normal-3780.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3780.png,Normal/images/Normal-3780.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3780.png,Normal/masks/Normal-3780.png,True,False,299,299,L,106.81,b5e7d97dba6cb4912ab7908bb288ef68
Normal-3781.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3781.png,Normal/images/Normal-3781.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3781.png,Normal/masks/Normal-3781.png,True,False,299,299,L,111.03,a147ece3e39df036319453a2904ae0e9
Normal-3782.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3782.png,Normal/images/Normal-3782.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3782.png,Normal/masks/Normal-3782.png,True,False,299,299,L,102.1,980c3aa8b3fdc242c693deac97ad3616
Normal-3783.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3783.png,Normal/images/Normal-3783.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3783.png,Normal/masks/Normal-3783.png,True,False,299,299,L,122.42,2ca857c989d1deb344baa47ed3f34da0
Normal-3784.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3784.png,Normal/images/Normal-3784.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3784.png,Normal/masks/Normal-3784.png,True,False,299,299,L,183.34,79ea168f598f7ffc623deacc258467de
Normal-3785.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3785.png,Normal/images/Normal-3785.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3785.png,Normal/masks/Normal-3785.png,True,False,299,299,L,151.96,375e092699cf603dcccb10f46ce2fd1d
Normal-3786.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3786.png,Normal/images/Normal-3786.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3786.png,Normal/masks/Normal-3786.png,True,False,299,299,L,116.31,58839faf855a03227aff7f93919d49f3
Normal-3787.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3787.png,Normal/images/Normal-3787.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3787.png,Normal/masks/Normal-3787.png,True,False,299,299,L,134.29,95f3b31684111b57ce7fc335c5615716
Normal-3788.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3788.png,Normal/images/Normal-3788.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3788.png,Normal/masks/Normal-3788.png,True,False,299,299,L,112.0,05aadf0576160257c5466053d6c7f42a
Normal-3789.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3789.png,Normal/images/Normal-3789.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3789.png,Normal/masks/Normal-3789.png,True,False,299,299,L,158.11,057bd5840a4451f07bed194adaa2d245
Normal-379.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-379.png,Normal/images/Normal-379.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-379.png,Normal/masks/Normal-379.png,True,False,299,299,L,115.5,37e2757b71dccf1c4ca8ee1982bc4876
Normal-3790.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3790.png,Normal/images/Normal-3790.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3790.png,Normal/masks/Normal-3790.png,True,False,299,299,L,130.29,0a1338191a7c0940380a1b38adf3e915
Normal-3791.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3791.png,Normal/images/Normal-3791.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3791.png,Normal/masks/Normal-3791.png,True,False,299,299,L,118.67,84355c3e3bf54de9feceb92ea62ae2cf
Normal-3792.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3792.png,Normal/images/Normal-3792.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3792.png,Normal/masks/Normal-3792.png,True,False,299,299,L,103.3,ff0b7a07ac1497539e98e2c80f37180d
Normal-3793.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3793.png,Normal/images/Normal-3793.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3793.png,Normal/masks/Normal-3793.png,True,False,299,299,L,114.7,459cc476d6c45803e8e969230cdd6525
Normal-3794.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3794.png,Normal/images/Normal-3794.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3794.png,Normal/masks/Normal-3794.png,True,False,299,299,L,129.18,6b78e5ff6f61c8954ed521e629911a70
Normal-3795.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3795.png,Normal/images/Normal-3795.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3795.png,Normal/masks/Normal-3795.png,True,False,299,299,L,152.57,f0876b68cb8a74f9eec514db05c9bf1f
Normal-3796.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3796.png,Normal/images/Normal-3796.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3796.png,Normal/masks/Normal-3796.png,True,False,299,299,L,108.73,15b7c5f6475510a3bffcbb893de8c2be
Normal-3797.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3797.png,Normal/images/Normal-3797.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3797.png,Normal/masks/Normal-3797.png,True,False,299,299,L,162.23,9024e2342fa648161f57bd7e9f7f694d
Normal-3798.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3798.png,Normal/images/Normal-3798.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3798.png,Normal/masks/Normal-3798.png,True,False,299,299,L,110.14,e87679e4177e3bde49a049d5675a55ba
Normal-3799.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3799.png,Normal/images/Normal-3799.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3799.png,Normal/masks/Normal-3799.png,True,False,299,299,L,156.34,98ae5f21b2b222ba08d505fbd05244d0
Normal-38.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-38.png,Normal/images/Normal-38.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-38.png,Normal/masks/Normal-38.png,True,False,299,299,L,174.66,cc4cb31a0d42b2c756763b28dbe4bf07
Normal-380.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-380.png,Normal/images/Normal-380.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-380.png,Normal/masks/Normal-380.png,True,False,299,299,L,126.16,1249f7918488149121135245013cf714
Normal-3800.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3800.png,Normal/images/Normal-3800.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3800.png,Normal/masks/Normal-3800.png,True,False,299,299,L,150.41,ed5f606eacd140106178ce99efc786fa
Normal-3801.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3801.png,Normal/images/Normal-3801.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3801.png,Normal/masks/Normal-3801.png,True,False,299,299,L,163.77,1487bba6fd9438419295e447d8176a1b
Normal-3802.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3802.png,Normal/images/Normal-3802.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3802.png,Normal/masks/Normal-3802.png,True,False,299,299,L,124.27,646e03aa72a71053f02d8df6af645fc5
Normal-3803.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3803.png,Normal/images/Normal-3803.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3803.png,Normal/masks/Normal-3803.png,True,False,299,299,L,128.66,2305885e001afbe424bd7e4c3051755b
Normal-3804.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3804.png,Normal/images/Normal-3804.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3804.png,Normal/masks/Normal-3804.png,True,False,299,299,L,128.93,dceabaecd6b214952e0e7535600904b1
Normal-3805.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3805.png,Normal/images/Normal-3805.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3805.png,Normal/masks/Normal-3805.png,True,False,299,299,L,138.76,8fa3dbb4b680c03abf317a4b83c0b09a
Normal-3806.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3806.png,Normal/images/Normal-3806.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3806.png,Normal/masks/Normal-3806.png,True,False,299,299,L,169.54,67a4aea40defb1d4f3923881a4bf258e
Normal-3807.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3807.png,Normal/images/Normal-3807.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3807.png,Normal/masks/Normal-3807.png,True,False,299,299,L,161.98,cc6ca418d09c16a530c7efcefbbeb56d
Normal-3808.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3808.png,Normal/images/Normal-3808.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3808.png,Normal/masks/Normal-3808.png,True,False,299,299,L,129.46,42fdf2f862b7a2fcacd1add1a0a5cf7a
Normal-3809.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3809.png,Normal/images/Normal-3809.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3809.png,Normal/masks/Normal-3809.png,True,False,299,299,L,107.99,63d7691c6c4cc47c6d923df6490866d3
Normal-381.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-381.png,Normal/images/Normal-381.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-381.png,Normal/masks/Normal-381.png,True,False,299,299,L,129.08,b08789929789ac3734311a67853dc106
Normal-3810.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3810.png,Normal/images/Normal-3810.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3810.png,Normal/masks/Normal-3810.png,True,False,299,299,L,142.91,d2ad2864274502577a56f75ad706f035
Normal-3811.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3811.png,Normal/images/Normal-3811.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3811.png,Normal/masks/Normal-3811.png,True,False,299,299,L,162.3,0ce9998b5a3f25a025c52241b1b86cb4
Normal-3812.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3812.png,Normal/images/Normal-3812.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3812.png,Normal/masks/Normal-3812.png,True,False,299,299,L,107.64,032ee3bb2770d70eaaf13a796f925ce6
Normal-3813.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3813.png,Normal/images/Normal-3813.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3813.png,Normal/masks/Normal-3813.png,True,False,299,299,L,142.97,5d324a0b234da3c5c7704714a063357d
Normal-3814.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3814.png,Normal/images/Normal-3814.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3814.png,Normal/masks/Normal-3814.png,True,False,299,299,L,171.28,bd298f4b4c1857f9dfc4c1e3be2229cf
Normal-3815.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3815.png,Normal/images/Normal-3815.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3815.png,Normal/masks/Normal-3815.png,True,False,299,299,L,100.54,72d5014db332468129b0b4b682ebc55f
Normal-3816.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3816.png,Normal/images/Normal-3816.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3816.png,Normal/masks/Normal-3816.png,True,False,299,299,L,120.77,fad9cc92a7c121e1514e99a81c9ef4e3
Normal-3817.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3817.png,Normal/images/Normal-3817.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3817.png,Normal/masks/Normal-3817.png,True,False,299,299,L,161.46,e21c16d7ba740931e5d84cee383c44df
Normal-3818.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3818.png,Normal/images/Normal-3818.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3818.png,Normal/masks/Normal-3818.png,True,False,299,299,L,145.38,0704b6372ad41034fe9821055c0436f9
Normal-3819.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3819.png,Normal/images/Normal-3819.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3819.png,Normal/masks/Normal-3819.png,True,False,299,299,L,124.07,ae477ebdfdc2c57cfbfe5d69c422a9e5
Normal-382.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-382.png,Normal/images/Normal-382.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-382.png,Normal/masks/Normal-382.png,True,False,299,299,L,140.1,c7c6a15e18acfa4edaaab2be8545bba5
Normal-3820.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3820.png,Normal/images/Normal-3820.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3820.png,Normal/masks/Normal-3820.png,True,False,299,299,L,104.62,6c4fb10f3983b44b7729c06a0716a29f
Normal-3821.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3821.png,Normal/images/Normal-3821.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3821.png,Normal/masks/Normal-3821.png,True,False,299,299,L,133.6,1eca08bd5f0d35cce6065b8c54c2fc56
Normal-3822.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3822.png,Normal/images/Normal-3822.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3822.png,Normal/masks/Normal-3822.png,True,False,299,299,L,151.99,b7fcea6a3afba6d1dfd6ad58a23cbcd8
Normal-3823.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3823.png,Normal/images/Normal-3823.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3823.png,Normal/masks/Normal-3823.png,True,False,299,299,L,121.9,03fef5a2274cb3d0b2a130a65596e08b
Normal-3824.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3824.png,Normal/images/Normal-3824.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3824.png,Normal/masks/Normal-3824.png,True,False,299,299,L,130.65,e9dbd9af6ea58eb53189bed5fcd0fa8c
Normal-3825.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3825.png,Normal/images/Normal-3825.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3825.png,Normal/masks/Normal-3825.png,True,False,299,299,L,112.39,d1fa24f8dc0db968b4a5dfdcb3b9c719
Normal-3826.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3826.png,Normal/images/Normal-3826.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3826.png,Normal/masks/Normal-3826.png,True,False,299,299,L,121.68,5f816b89f918cc60f0645eceff8335ee
Normal-3827.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3827.png,Normal/images/Normal-3827.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3827.png,Normal/masks/Normal-3827.png,True,False,299,299,L,183.78,dda1814113e10764f389471317c7014f
Normal-3828.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3828.png,Normal/images/Normal-3828.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3828.png,Normal/masks/Normal-3828.png,True,False,299,299,L,109.46,6ffac54d9d6e8ec1783b53d7be2853ad
Normal-3829.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3829.png,Normal/images/Normal-3829.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3829.png,Normal/masks/Normal-3829.png,True,False,299,299,L,183.66,ae53fa28ceca368523d90cf13ded4cfb
Normal-383.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-383.png,Normal/images/Normal-383.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-383.png,Normal/masks/Normal-383.png,True,False,299,299,L,120.58,43cb8d49b792d6b67da5c1ab48be94b2
Normal-3830.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3830.png,Normal/images/Normal-3830.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3830.png,Normal/masks/Normal-3830.png,True,False,299,299,L,174.07,f58ef2af73b8cf7fa5da416aedfe311e
Normal-3831.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3831.png,Normal/images/Normal-3831.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3831.png,Normal/masks/Normal-3831.png,True,False,299,299,L,131.51,0fc7516cd4216463aef5585a0bef1a96
Normal-3832.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3832.png,Normal/images/Normal-3832.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3832.png,Normal/masks/Normal-3832.png,True,False,299,299,L,163.59,22a4beae7d41e673cfaead1db2ddea5e
Normal-3833.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3833.png,Normal/images/Normal-3833.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3833.png,Normal/masks/Normal-3833.png,True,False,299,299,L,177.99,975186c4ec8611787b9bb9df7c07f368
Normal-3834.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3834.png,Normal/images/Normal-3834.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3834.png,Normal/masks/Normal-3834.png,True,False,299,299,L,113.45,168687ff4a94ab1b5f7ffddcfdb1791a
Normal-3835.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3835.png,Normal/images/Normal-3835.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3835.png,Normal/masks/Normal-3835.png,True,False,299,299,L,115.54,44e0133c8a43c82c6f32204b9d4fa3d4
Normal-3836.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3836.png,Normal/images/Normal-3836.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3836.png,Normal/masks/Normal-3836.png,True,False,299,299,L,114.18,a5f07534e960bf1ea5af1aab6ae45d6f
Normal-3837.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3837.png,Normal/images/Normal-3837.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3837.png,Normal/masks/Normal-3837.png,True,False,299,299,L,116.61,9e232403a43dc654a02f7fd55dea3569
Normal-3838.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3838.png,Normal/images/Normal-3838.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3838.png,Normal/masks/Normal-3838.png,True,False,299,299,L,160.26,63c9d137c73cfc505263d63c4afa5ed0
Normal-3839.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3839.png,Normal/images/Normal-3839.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3839.png,Normal/masks/Normal-3839.png,True,False,299,299,L,84.62,668d38b3b9c22f306e9496e10fe0ab23
Normal-384.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-384.png,Normal/images/Normal-384.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-384.png,Normal/masks/Normal-384.png,True,False,299,299,L,119.98,3b1977868d6f2e52f89a74fa885fb5f7
Normal-3840.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3840.png,Normal/images/Normal-3840.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3840.png,Normal/masks/Normal-3840.png,True,False,299,299,L,154.64,a5bed80aeb0fddca7ace3344124a9baf
Normal-3841.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3841.png,Normal/images/Normal-3841.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3841.png,Normal/masks/Normal-3841.png,True,False,299,299,L,102.41,86d7fd1fd3b62bba4936709dc3478dd8
Normal-3842.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3842.png,Normal/images/Normal-3842.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3842.png,Normal/masks/Normal-3842.png,True,False,299,299,L,114.51,cd702a9e571cf8c8b590eb00aee2301a
Normal-3843.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3843.png,Normal/images/Normal-3843.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3843.png,Normal/masks/Normal-3843.png,True,False,299,299,L,117.9,acd1f40c66da8fd516e8e1f4d4ee7c3a
Normal-3844.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3844.png,Normal/images/Normal-3844.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3844.png,Normal/masks/Normal-3844.png,True,False,299,299,L,109.61,d64717b289cd6b5e761e02ebd08de13d
Normal-3845.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3845.png,Normal/images/Normal-3845.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3845.png,Normal/masks/Normal-3845.png,True,False,299,299,L,108.15,9429020b4b1c785f1b03c9b7cd22c80f
Normal-3846.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3846.png,Normal/images/Normal-3846.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3846.png,Normal/masks/Normal-3846.png,True,False,299,299,L,169.34,dad7f43043965e01c0fb78dcbcd6ae8c
Normal-3847.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3847.png,Normal/images/Normal-3847.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3847.png,Normal/masks/Normal-3847.png,True,False,299,299,L,171.38,0bdd167dcb2f3cd85232a3a0ea86688e
Normal-3848.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3848.png,Normal/images/Normal-3848.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3848.png,Normal/masks/Normal-3848.png,True,False,299,299,L,161.01,804d4f6f9dd019a6fc6c4cdacfe437f4
Normal-3849.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3849.png,Normal/images/Normal-3849.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3849.png,Normal/masks/Normal-3849.png,True,False,299,299,L,139.34,eaa521ac6dd864c155d84ac5019a6c54
Normal-385.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-385.png,Normal/images/Normal-385.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-385.png,Normal/masks/Normal-385.png,True,False,299,299,L,120.26,d80e115e3a842338d76f8216efeb1004
Normal-3850.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3850.png,Normal/images/Normal-3850.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3850.png,Normal/masks/Normal-3850.png,True,False,299,299,L,132.89,a0cf6e933abb55ec6e308f2e92d4c5c5
Normal-3851.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3851.png,Normal/images/Normal-3851.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3851.png,Normal/masks/Normal-3851.png,True,False,299,299,L,138.44,f42687241c3db5307cdf27c56a55068a
Normal-3852.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3852.png,Normal/images/Normal-3852.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3852.png,Normal/masks/Normal-3852.png,True,False,299,299,L,121.25,f58cddee541cc2cf32bdc52753100274
Normal-3853.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3853.png,Normal/images/Normal-3853.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3853.png,Normal/masks/Normal-3853.png,True,False,299,299,L,158.85,b39508ba18860acf53575fd778e1c7ea
Normal-3854.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3854.png,Normal/images/Normal-3854.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3854.png,Normal/masks/Normal-3854.png,True,False,299,299,L,157.94,76f7de9895ef69227bcbaf16b51b058b
Normal-3855.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3855.png,Normal/images/Normal-3855.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3855.png,Normal/masks/Normal-3855.png,True,False,299,299,L,104.06,7b403e35b02c82df1e304e25b3d3e154
Normal-3856.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3856.png,Normal/images/Normal-3856.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3856.png,Normal/masks/Normal-3856.png,True,False,299,299,L,159.8,ca1c4e6998691334849422ef6a3f14ad
Normal-3857.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3857.png,Normal/images/Normal-3857.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3857.png,Normal/masks/Normal-3857.png,True,False,299,299,L,154.39,e2f4d1db5beaec73e37566fcd8e12dc8
Normal-3858.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3858.png,Normal/images/Normal-3858.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3858.png,Normal/masks/Normal-3858.png,True,False,299,299,L,140.33,e9625df0c6b7422804e320c05f84a69c
Normal-3859.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3859.png,Normal/images/Normal-3859.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3859.png,Normal/masks/Normal-3859.png,True,False,299,299,L,70.37,4c9b254308cc99e2fb7ddc6c6f32d2a6
Normal-386.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-386.png,Normal/images/Normal-386.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-386.png,Normal/masks/Normal-386.png,True,False,299,299,L,112.66,ee2190562e88c386afcc36e3de1709ea
Normal-3860.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3860.png,Normal/images/Normal-3860.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3860.png,Normal/masks/Normal-3860.png,True,False,299,299,L,128.5,82e2bc72c8f442cd1317c8a3517116b6
Normal-3861.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3861.png,Normal/images/Normal-3861.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3861.png,Normal/masks/Normal-3861.png,True,False,299,299,L,132.74,9b40dc5fa2e9d1d888a36ae5100ce75f
Normal-3862.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3862.png,Normal/images/Normal-3862.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3862.png,Normal/masks/Normal-3862.png,True,False,299,299,L,131.41,f5273943554040acbd52684c4789653e
Normal-3863.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3863.png,Normal/images/Normal-3863.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3863.png,Normal/masks/Normal-3863.png,True,False,299,299,L,137.59,c33a8be9207554e043a0793a6d17d62b
Normal-3864.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3864.png,Normal/images/Normal-3864.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3864.png,Normal/masks/Normal-3864.png,True,False,299,299,L,160.26,3ab3c8b18f9780ba351bb9f22704f9d5
Normal-3865.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3865.png,Normal/images/Normal-3865.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3865.png,Normal/masks/Normal-3865.png,True,False,299,299,L,122.51,c6ca9069c761283822fb4c7a2032ffc4
Normal-3866.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3866.png,Normal/images/Normal-3866.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3866.png,Normal/masks/Normal-3866.png,True,False,299,299,L,101.73,72c3642fe5cc230c62bec738eeeb1a3a
Normal-3867.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3867.png,Normal/images/Normal-3867.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3867.png,Normal/masks/Normal-3867.png,True,False,299,299,L,141.31,927b59104667b9107ad0ca71e121aa9e
Normal-3868.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3868.png,Normal/images/Normal-3868.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3868.png,Normal/masks/Normal-3868.png,True,False,299,299,L,139.46,50c9d5bd1b4d15d96dd9e5aa126eccc0
Normal-3869.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3869.png,Normal/images/Normal-3869.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3869.png,Normal/masks/Normal-3869.png,True,False,299,299,L,119.1,2b2376e497fa88d2eaaf013b9e022e7a
Normal-387.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-387.png,Normal/images/Normal-387.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-387.png,Normal/masks/Normal-387.png,True,False,299,299,L,131.0,bd1a8f7488f227e510a4a858488d7fe3
Normal-3870.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3870.png,Normal/images/Normal-3870.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3870.png,Normal/masks/Normal-3870.png,True,False,299,299,L,146.44,f4899f935203f6a3dfd58ff8c09810ed
Normal-3871.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3871.png,Normal/images/Normal-3871.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3871.png,Normal/masks/Normal-3871.png,True,False,299,299,L,100.27,5001589a13868dae4d5c6af98645689a
Normal-3872.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3872.png,Normal/images/Normal-3872.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3872.png,Normal/masks/Normal-3872.png,True,False,299,299,L,149.35,4a68e63a67ac3e65325a601e9da8b45d
Normal-3873.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3873.png,Normal/images/Normal-3873.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3873.png,Normal/masks/Normal-3873.png,True,False,299,299,L,118.01,0d8c0566be9494a3a737cc8ea159f939
Normal-3874.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3874.png,Normal/images/Normal-3874.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3874.png,Normal/masks/Normal-3874.png,True,False,299,299,L,142.11,9f0a0590df6d7f48affdcac814a96365
Normal-3875.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3875.png,Normal/images/Normal-3875.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3875.png,Normal/masks/Normal-3875.png,True,False,299,299,L,170.58,64d30113cbdbcedcedccceaa97b8a38b
Normal-3876.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3876.png,Normal/images/Normal-3876.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3876.png,Normal/masks/Normal-3876.png,True,False,299,299,L,127.13,01c51d70a9fb3a85f0127a357c091fc3
Normal-3877.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3877.png,Normal/images/Normal-3877.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3877.png,Normal/masks/Normal-3877.png,True,False,299,299,L,121.26,0b9b38845050418c84269c8f4bfd7e71
Normal-3878.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3878.png,Normal/images/Normal-3878.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3878.png,Normal/masks/Normal-3878.png,True,False,299,299,L,127.51,af2c38d1d6d7f0d42c0267c1e1c4f5e6
Normal-3879.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3879.png,Normal/images/Normal-3879.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3879.png,Normal/masks/Normal-3879.png,True,False,299,299,L,120.95,77661b8f386c7f02cf8af698df503d9e
Normal-388.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-388.png,Normal/images/Normal-388.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-388.png,Normal/masks/Normal-388.png,True,False,299,299,L,112.42,bc4203ad471f5d1d3c25f4648eb741bb
Normal-3880.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3880.png,Normal/images/Normal-3880.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3880.png,Normal/masks/Normal-3880.png,True,False,299,299,L,128.46,2bf958d2b5b6e511d2510ab079b81798
Normal-3881.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3881.png,Normal/images/Normal-3881.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3881.png,Normal/masks/Normal-3881.png,True,False,299,299,L,163.43,d64c8ce63600c0bcaff3fe35ff5c23d8
Normal-3882.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3882.png,Normal/images/Normal-3882.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3882.png,Normal/masks/Normal-3882.png,True,False,299,299,L,118.19,92d15bf3a986eea6c39e05262ead5683
Normal-3883.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3883.png,Normal/images/Normal-3883.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3883.png,Normal/masks/Normal-3883.png,True,False,299,299,L,105.48,e97427b4c7d307fa5b6930fcb24be17f
Normal-3884.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3884.png,Normal/images/Normal-3884.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3884.png,Normal/masks/Normal-3884.png,True,False,299,299,L,123.72,e3af32b3005947a3f4b91a5da2f03d7d
Normal-3885.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3885.png,Normal/images/Normal-3885.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3885.png,Normal/masks/Normal-3885.png,True,False,299,299,L,88.69,0c08e7231936c28d0b57a3f190233207
Normal-3886.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3886.png,Normal/images/Normal-3886.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3886.png,Normal/masks/Normal-3886.png,True,False,299,299,L,134.66,736431ba62d5409baa9d8c3c3e5f76d6
Normal-3887.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3887.png,Normal/images/Normal-3887.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3887.png,Normal/masks/Normal-3887.png,True,False,299,299,L,129.58,bdaf4fc84c953a6ecbe25228fd44c5f9
Normal-3888.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3888.png,Normal/images/Normal-3888.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3888.png,Normal/masks/Normal-3888.png,True,False,299,299,L,159.0,e7671c56858e699f4a31b2b4d184e9be
Normal-3889.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3889.png,Normal/images/Normal-3889.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3889.png,Normal/masks/Normal-3889.png,True,False,299,299,L,121.07,187dbf0f5a87d42badaf6cca28c4afb6
Normal-389.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-389.png,Normal/images/Normal-389.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-389.png,Normal/masks/Normal-389.png,True,False,299,299,L,119.8,ab3df53336567516db1cf34e0668fac7
Normal-3890.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3890.png,Normal/images/Normal-3890.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3890.png,Normal/masks/Normal-3890.png,True,False,299,299,L,122.67,2f91d8976ff9f2e07e01ebd07b59202b
Normal-3891.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3891.png,Normal/images/Normal-3891.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3891.png,Normal/masks/Normal-3891.png,True,False,299,299,L,92.27,f92d8e58f18abd695d39be813c2a1bfb
Normal-3892.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3892.png,Normal/images/Normal-3892.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3892.png,Normal/masks/Normal-3892.png,True,False,299,299,L,137.67,d3308dbe02d60c296a7cf6ec12b7e6d4
Normal-3893.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3893.png,Normal/images/Normal-3893.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3893.png,Normal/masks/Normal-3893.png,True,False,299,299,L,104.42,d84e94e89ac1fa601c6b279c52eb44f9
Normal-3894.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3894.png,Normal/images/Normal-3894.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3894.png,Normal/masks/Normal-3894.png,True,False,299,299,L,154.01,01ceb10e03d42983625b2203df802de3
Normal-3895.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3895.png,Normal/images/Normal-3895.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3895.png,Normal/masks/Normal-3895.png,True,False,299,299,L,115.38,7e53383966a7c7109160ce5268b89638
Normal-3896.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3896.png,Normal/images/Normal-3896.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3896.png,Normal/masks/Normal-3896.png,True,False,299,299,L,106.78,06092be212ac7265c34559c109824d0d
Normal-3897.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3897.png,Normal/images/Normal-3897.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3897.png,Normal/masks/Normal-3897.png,True,False,299,299,L,105.59,c8182ba7dde2a3ff51e0ace984f441ba
Normal-3898.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3898.png,Normal/images/Normal-3898.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3898.png,Normal/masks/Normal-3898.png,True,False,299,299,L,159.36,0ef2c29db1fdb45454a306bc4570f77a
Normal-3899.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3899.png,Normal/images/Normal-3899.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3899.png,Normal/masks/Normal-3899.png,True,False,299,299,L,107.93,b8fc07b4b241c6bfffc9167a87d5a719
Normal-39.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-39.png,Normal/images/Normal-39.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-39.png,Normal/masks/Normal-39.png,True,False,299,299,L,112.23,d3ace391749887dfc643336aa0a8a7d8
Normal-390.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-390.png,Normal/images/Normal-390.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-390.png,Normal/masks/Normal-390.png,True,False,299,299,L,102.9,a26f488e328bc05be872dba57c88b3b2
Normal-3900.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3900.png,Normal/images/Normal-3900.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3900.png,Normal/masks/Normal-3900.png,True,False,299,299,L,171.92,f5bc8958c635397dca3924046c43253c
Normal-3901.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3901.png,Normal/images/Normal-3901.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3901.png,Normal/masks/Normal-3901.png,True,False,299,299,L,144.27,548a0b7132120004ee9d386479b8506e
Normal-3902.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3902.png,Normal/images/Normal-3902.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3902.png,Normal/masks/Normal-3902.png,True,False,299,299,L,122.02,ec387a9e56bebcaa77d2eb21873324ab
Normal-3903.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3903.png,Normal/images/Normal-3903.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3903.png,Normal/masks/Normal-3903.png,True,False,299,299,L,136.23,afeffb3411d9f5e4aafa1a96eb0245c8
Normal-3904.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3904.png,Normal/images/Normal-3904.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3904.png,Normal/masks/Normal-3904.png,True,False,299,299,L,98.88,c1ef47caa8cefd9881143e239ac985d6
Normal-3905.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3905.png,Normal/images/Normal-3905.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3905.png,Normal/masks/Normal-3905.png,True,False,299,299,L,106.81,f7718233692c34f9dbbdd8d3db56b2f1
Normal-3906.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3906.png,Normal/images/Normal-3906.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3906.png,Normal/masks/Normal-3906.png,True,False,299,299,L,131.97,8d56d564a83c5cae65ec21710d8746c3
Normal-3907.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3907.png,Normal/images/Normal-3907.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3907.png,Normal/masks/Normal-3907.png,True,False,299,299,L,111.56,dcbe9c4ec616332f8133bb435434a61a
Normal-3908.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3908.png,Normal/images/Normal-3908.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3908.png,Normal/masks/Normal-3908.png,True,False,299,299,L,125.51,1adeaa5487da4fdd9bb41f92dcab7873
Normal-3909.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3909.png,Normal/images/Normal-3909.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3909.png,Normal/masks/Normal-3909.png,True,False,299,299,L,148.7,47b0fbb4759301908cb46fb8c246aced
Normal-391.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-391.png,Normal/images/Normal-391.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-391.png,Normal/masks/Normal-391.png,True,False,299,299,L,106.62,ae05e10be806a7067e68eaa09eafafda
Normal-3910.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3910.png,Normal/images/Normal-3910.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3910.png,Normal/masks/Normal-3910.png,True,False,299,299,L,147.51,f61403e9574733ebecf0a697fd48092f
Normal-3911.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3911.png,Normal/images/Normal-3911.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3911.png,Normal/masks/Normal-3911.png,True,False,299,299,L,138.46,583a7accc57ff0d8853bee1fee49d99e
Normal-3912.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3912.png,Normal/images/Normal-3912.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3912.png,Normal/masks/Normal-3912.png,True,False,299,299,L,129.93,72ef9c31702573c5b921ce0c5ba529ec
Normal-3913.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3913.png,Normal/images/Normal-3913.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3913.png,Normal/masks/Normal-3913.png,True,False,299,299,L,131.49,097dda4cecc43e2cf65fed0e32dd6c6c
Normal-3914.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3914.png,Normal/images/Normal-3914.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3914.png,Normal/masks/Normal-3914.png,True,False,299,299,L,109.33,7cff12d495a2c9999c93d5b4399126ca
Normal-3915.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3915.png,Normal/images/Normal-3915.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3915.png,Normal/masks/Normal-3915.png,True,False,299,299,L,129.42,424809159c1c3f9bb714c7704dcefd80
Normal-3916.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3916.png,Normal/images/Normal-3916.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3916.png,Normal/masks/Normal-3916.png,True,False,299,299,L,115.53,a1346583a74b4bad7f94273741b3631a
Normal-3917.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3917.png,Normal/images/Normal-3917.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3917.png,Normal/masks/Normal-3917.png,True,False,299,299,L,141.83,b299131e123cc87dcb96441c25ef33b4
Normal-3918.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3918.png,Normal/images/Normal-3918.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3918.png,Normal/masks/Normal-3918.png,True,False,299,299,L,109.14,30200b9d89fd159d7d54cc39ae77428c
Normal-3919.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3919.png,Normal/images/Normal-3919.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3919.png,Normal/masks/Normal-3919.png,True,False,299,299,L,154.78,573b3334c01de6333467d2cf0692655a
Normal-392.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-392.png,Normal/images/Normal-392.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-392.png,Normal/masks/Normal-392.png,True,False,299,299,L,120.61,11af1418de2533099f25aa103bcbc83e
Normal-3920.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3920.png,Normal/images/Normal-3920.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3920.png,Normal/masks/Normal-3920.png,True,False,299,299,L,139.64,4d7a0eb1089e388d94e6bfeef97be2bd
Normal-3921.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3921.png,Normal/images/Normal-3921.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3921.png,Normal/masks/Normal-3921.png,True,False,299,299,L,145.57,0ffcb25c995db2d6f00d1e24b9328535
Normal-3922.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3922.png,Normal/images/Normal-3922.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3922.png,Normal/masks/Normal-3922.png,True,False,299,299,L,158.39,dd13df795072608d444cb10702459ba9
Normal-3923.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3923.png,Normal/images/Normal-3923.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3923.png,Normal/masks/Normal-3923.png,True,False,299,299,L,148.45,48f3488bd970b4764742be4db09d9f2c
Normal-3924.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3924.png,Normal/images/Normal-3924.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3924.png,Normal/masks/Normal-3924.png,True,False,299,299,L,164.07,4b58d3f3ac5cf7ae7f2bb7175bdc06f5
Normal-3925.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3925.png,Normal/images/Normal-3925.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3925.png,Normal/masks/Normal-3925.png,True,False,299,299,L,176.26,8009497ae7c711ac9fea1527b3255ec5
Normal-3926.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3926.png,Normal/images/Normal-3926.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3926.png,Normal/masks/Normal-3926.png,True,False,299,299,L,137.39,9e1b16fe4602423a1d9c48b3e1f09cda
Normal-3927.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3927.png,Normal/images/Normal-3927.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3927.png,Normal/masks/Normal-3927.png,True,False,299,299,L,158.44,3ea24de2f54061787ce58b8ac586055c
Normal-3928.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3928.png,Normal/images/Normal-3928.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3928.png,Normal/masks/Normal-3928.png,True,False,299,299,L,148.14,401cd042584d6069ef4828cfd41c9c6c
Normal-3929.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3929.png,Normal/images/Normal-3929.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3929.png,Normal/masks/Normal-3929.png,True,False,299,299,L,107.03,3f263992a1ae0307e923eca468e27dbd
Normal-393.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-393.png,Normal/images/Normal-393.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-393.png,Normal/masks/Normal-393.png,True,False,299,299,L,138.87,5a4d87d00d3493f2dda5ff3ca62706d1
Normal-3930.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3930.png,Normal/images/Normal-3930.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3930.png,Normal/masks/Normal-3930.png,True,False,299,299,L,126.94,7eef41656ce29b5148e65bb62202daf7
Normal-3931.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3931.png,Normal/images/Normal-3931.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3931.png,Normal/masks/Normal-3931.png,True,False,299,299,L,151.07,8b0b86120094ea283d907d779caefdf8
Normal-3932.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3932.png,Normal/images/Normal-3932.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3932.png,Normal/masks/Normal-3932.png,True,False,299,299,L,127.81,26f7ff91dd17b1afb24dc4976be24555
Normal-3933.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3933.png,Normal/images/Normal-3933.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3933.png,Normal/masks/Normal-3933.png,True,False,299,299,L,166.83,4ca4c4b05304417eeb9a2263e854f564
Normal-3934.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3934.png,Normal/images/Normal-3934.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3934.png,Normal/masks/Normal-3934.png,True,False,299,299,L,123.47,39e19e0d5a0a903224025f8c9f27867d
Normal-3935.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3935.png,Normal/images/Normal-3935.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3935.png,Normal/masks/Normal-3935.png,True,False,299,299,L,134.47,71a115209b07ae55756a74fd82907bff
Normal-3936.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3936.png,Normal/images/Normal-3936.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3936.png,Normal/masks/Normal-3936.png,True,False,299,299,L,117.71,ba5c4218996c598ce97d4ea62e2557b8
Normal-3937.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3937.png,Normal/images/Normal-3937.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3937.png,Normal/masks/Normal-3937.png,True,False,299,299,L,118.44,d3dee18bf914d96141586cc6d68771bf
Normal-3938.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3938.png,Normal/images/Normal-3938.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3938.png,Normal/masks/Normal-3938.png,True,False,299,299,L,140.92,8950520ce1d98c4ad1f653c8bdfa8116
Normal-3939.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3939.png,Normal/images/Normal-3939.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3939.png,Normal/masks/Normal-3939.png,True,False,299,299,L,142.59,be8e7381a06d364b36ad6052ce4a3961
Normal-394.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-394.png,Normal/images/Normal-394.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-394.png,Normal/masks/Normal-394.png,True,False,299,299,L,131.39,d0a2a6f5d594259e6c4a02f747598fc8
Normal-3940.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3940.png,Normal/images/Normal-3940.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3940.png,Normal/masks/Normal-3940.png,True,False,299,299,L,129.74,bb273af53739bf092365201768edea3a
Normal-3941.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3941.png,Normal/images/Normal-3941.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3941.png,Normal/masks/Normal-3941.png,True,False,299,299,L,104.25,bbc6bb11821f00d2eb7382422919ac43
Normal-3942.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3942.png,Normal/images/Normal-3942.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3942.png,Normal/masks/Normal-3942.png,True,False,299,299,L,110.36,c7fb5f9b99fc74c5687c24f1b365ce37
Normal-3943.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3943.png,Normal/images/Normal-3943.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3943.png,Normal/masks/Normal-3943.png,True,False,299,299,L,159.18,6cece664ff88ae7a0ecced8957f63492
Normal-3944.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3944.png,Normal/images/Normal-3944.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3944.png,Normal/masks/Normal-3944.png,True,False,299,299,L,155.07,e2eb2d5e13fa9e4fe46e8928289d0849
Normal-3945.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3945.png,Normal/images/Normal-3945.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3945.png,Normal/masks/Normal-3945.png,True,False,299,299,L,154.97,33d4edcc25b38830df95ff1c875a46e9
Normal-3946.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3946.png,Normal/images/Normal-3946.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3946.png,Normal/masks/Normal-3946.png,True,False,299,299,L,132.1,612c21d8f13e5ace71df63e4bce12b68
Normal-3947.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3947.png,Normal/images/Normal-3947.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3947.png,Normal/masks/Normal-3947.png,True,False,299,299,L,135.74,bf170ffb18d6b3fee06a520778cb4517
Normal-3948.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3948.png,Normal/images/Normal-3948.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3948.png,Normal/masks/Normal-3948.png,True,False,299,299,L,179.82,1ba36628e96c69266cddc9719d8738fe
Normal-3949.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3949.png,Normal/images/Normal-3949.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3949.png,Normal/masks/Normal-3949.png,True,False,299,299,L,120.07,957198230786b9c814b8acd19fe3dafb
Normal-395.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-395.png,Normal/images/Normal-395.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-395.png,Normal/masks/Normal-395.png,True,False,299,299,L,127.9,c04ba46c55fada5d0e6c236f23761e1e
Normal-3950.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3950.png,Normal/images/Normal-3950.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3950.png,Normal/masks/Normal-3950.png,True,False,299,299,L,105.36,b8380804d2f4f38cc13bae433d99dfe4
Normal-3951.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3951.png,Normal/images/Normal-3951.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3951.png,Normal/masks/Normal-3951.png,True,False,299,299,L,160.44,54bfa7d7739f660229d228c9a5d02542
Normal-3952.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3952.png,Normal/images/Normal-3952.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3952.png,Normal/masks/Normal-3952.png,True,False,299,299,L,130.82,8cd4fd581bcf76f7550ca214cc46f8c2
Normal-3953.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3953.png,Normal/images/Normal-3953.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3953.png,Normal/masks/Normal-3953.png,True,False,299,299,L,105.94,31df58adb3a3e2292b78804baff205ab
Normal-3954.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3954.png,Normal/images/Normal-3954.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3954.png,Normal/masks/Normal-3954.png,True,False,299,299,L,84.46,adc318a555963c647e6be8f0b337d4c4
Normal-3955.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3955.png,Normal/images/Normal-3955.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3955.png,Normal/masks/Normal-3955.png,True,False,299,299,L,106.37,4b8072c853fd17f5c3a01948c8156d4d
Normal-3956.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3956.png,Normal/images/Normal-3956.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3956.png,Normal/masks/Normal-3956.png,True,False,299,299,L,116.7,5b82f028480078a0664c6a30485974b9
Normal-3957.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3957.png,Normal/images/Normal-3957.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3957.png,Normal/masks/Normal-3957.png,True,False,299,299,L,148.32,59e03977d719c98abfce6744fb66e0cd
Normal-3958.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3958.png,Normal/images/Normal-3958.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3958.png,Normal/masks/Normal-3958.png,True,False,299,299,L,111.18,9f514a48c8ec9812e67fbcb80245a757
Normal-3959.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3959.png,Normal/images/Normal-3959.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3959.png,Normal/masks/Normal-3959.png,True,False,299,299,L,100.09,5cad33079091af3cb3a26652ea0902f3
Normal-396.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-396.png,Normal/images/Normal-396.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-396.png,Normal/masks/Normal-396.png,True,False,299,299,L,113.81,f49326a21db417eaf53d3279eed235a3
Normal-3960.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3960.png,Normal/images/Normal-3960.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3960.png,Normal/masks/Normal-3960.png,True,False,299,299,L,179.99,03b0e3409338afdaba58246b67031c69
Normal-3961.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3961.png,Normal/images/Normal-3961.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3961.png,Normal/masks/Normal-3961.png,True,False,299,299,L,116.2,726ce847515a355594ba03729f259a3d
Normal-3962.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3962.png,Normal/images/Normal-3962.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3962.png,Normal/masks/Normal-3962.png,True,False,299,299,L,166.63,254e9a89a8ae4c3ca87299918690d0e5
Normal-3963.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3963.png,Normal/images/Normal-3963.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3963.png,Normal/masks/Normal-3963.png,True,False,299,299,L,149.43,d013f3d48ca3df00c2aa7f12947ce36d
Normal-3964.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3964.png,Normal/images/Normal-3964.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3964.png,Normal/masks/Normal-3964.png,True,False,299,299,L,170.4,07221c1d58a733d3044eeca3504a7d49
Normal-3965.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3965.png,Normal/images/Normal-3965.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3965.png,Normal/masks/Normal-3965.png,True,False,299,299,L,116.8,a4027076f0f1b76e015cb92f7245ac38
Normal-3966.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3966.png,Normal/images/Normal-3966.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3966.png,Normal/masks/Normal-3966.png,True,False,299,299,L,139.34,921efe9565557f61a23e910ed83b3304
Normal-3967.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3967.png,Normal/images/Normal-3967.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3967.png,Normal/masks/Normal-3967.png,True,False,299,299,L,160.19,265248d507bc71d623f86441a13a555e
Normal-3968.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3968.png,Normal/images/Normal-3968.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3968.png,Normal/masks/Normal-3968.png,True,False,299,299,L,142.83,c7a4b7352aac87d81758b60e07deacab
Normal-3969.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3969.png,Normal/images/Normal-3969.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3969.png,Normal/masks/Normal-3969.png,True,False,299,299,L,111.53,0daed7331aa841425f08599cbb33469b
Normal-397.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-397.png,Normal/images/Normal-397.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-397.png,Normal/masks/Normal-397.png,True,False,299,299,L,107.43,c7cf7092baef4ea1c8be5f748e3b49aa
Normal-3970.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3970.png,Normal/images/Normal-3970.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3970.png,Normal/masks/Normal-3970.png,True,False,299,299,L,121.7,f244df9c21f6c7e4ea0ea0db3ce5c756
Normal-3971.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3971.png,Normal/images/Normal-3971.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3971.png,Normal/masks/Normal-3971.png,True,False,299,299,L,125.7,998cb2d5499283e6f2e8106035c63440
Normal-3972.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3972.png,Normal/images/Normal-3972.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3972.png,Normal/masks/Normal-3972.png,True,False,299,299,L,172.46,cbe3f046d10ac3500758022ebbb69854
Normal-3973.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3973.png,Normal/images/Normal-3973.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3973.png,Normal/masks/Normal-3973.png,True,False,299,299,L,96.69,a7d790096c0329118cd280f65f165dca
Normal-3974.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3974.png,Normal/images/Normal-3974.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3974.png,Normal/masks/Normal-3974.png,True,False,299,299,L,113.71,73e2ec6957e8841d74bf01373a089f49
Normal-3975.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3975.png,Normal/images/Normal-3975.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3975.png,Normal/masks/Normal-3975.png,True,False,299,299,L,131.78,60245fa62ebc1339a51932ceb5dc7bdd
Normal-3976.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3976.png,Normal/images/Normal-3976.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3976.png,Normal/masks/Normal-3976.png,True,False,299,299,L,135.65,b6163522e3859c36c7266f94625a462c
Normal-3977.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3977.png,Normal/images/Normal-3977.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3977.png,Normal/masks/Normal-3977.png,True,False,299,299,L,119.48,6e54fc1c75fb070819adb376cdc94b6e
Normal-3978.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3978.png,Normal/images/Normal-3978.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3978.png,Normal/masks/Normal-3978.png,True,False,299,299,L,116.92,3d31314ac338672d6c765f18b1f6966a
Normal-3979.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3979.png,Normal/images/Normal-3979.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3979.png,Normal/masks/Normal-3979.png,True,False,299,299,L,120.31,e44cb5a0b9ced6d41aa8079b922b381a
Normal-398.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-398.png,Normal/images/Normal-398.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-398.png,Normal/masks/Normal-398.png,True,False,299,299,L,135.42,ef1d8012e5190d616fb9d62bf12f8e08
Normal-3980.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3980.png,Normal/images/Normal-3980.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3980.png,Normal/masks/Normal-3980.png,True,False,299,299,L,173.77,8b7c627d080a69fb00f2db0a7e50a68d
Normal-3981.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3981.png,Normal/images/Normal-3981.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3981.png,Normal/masks/Normal-3981.png,True,False,299,299,L,121.48,8f518c6185336b59f0c1fbe3dc69cd43
Normal-3982.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3982.png,Normal/images/Normal-3982.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3982.png,Normal/masks/Normal-3982.png,True,False,299,299,L,155.59,a72497066880389311474a24cf40b028
Normal-3983.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3983.png,Normal/images/Normal-3983.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3983.png,Normal/masks/Normal-3983.png,True,False,299,299,L,128.16,408d5236af76572cd5b8da87a3ad0741
Normal-3984.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3984.png,Normal/images/Normal-3984.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3984.png,Normal/masks/Normal-3984.png,True,False,299,299,L,155.85,235627ab96aabb4752d0521da03f7258
Normal-3985.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3985.png,Normal/images/Normal-3985.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3985.png,Normal/masks/Normal-3985.png,True,False,299,299,L,157.67,33bc0d977eda916853e2e88a3e0170d8
Normal-3986.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3986.png,Normal/images/Normal-3986.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3986.png,Normal/masks/Normal-3986.png,True,False,299,299,L,113.26,dd7073987eaa059733593d92e71272c3
Normal-3987.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3987.png,Normal/images/Normal-3987.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3987.png,Normal/masks/Normal-3987.png,True,False,299,299,L,140.62,d29aa797b42a084ec0582157499d9956
Normal-3988.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3988.png,Normal/images/Normal-3988.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3988.png,Normal/masks/Normal-3988.png,True,False,299,299,L,96.31,74b0eccfd2316615e207f7e8d3d76ae4
Normal-3989.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3989.png,Normal/images/Normal-3989.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3989.png,Normal/masks/Normal-3989.png,True,False,299,299,L,125.01,72e93d23150e83c132dd2868f0c07a2a
Normal-399.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-399.png,Normal/images/Normal-399.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-399.png,Normal/masks/Normal-399.png,True,False,299,299,L,108.4,affd19e50c3309322cfda076ebb3bc72
Normal-3990.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3990.png,Normal/images/Normal-3990.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3990.png,Normal/masks/Normal-3990.png,True,False,299,299,L,103.08,de27e2b343aa92a80027c906a6dc45c9
Normal-3991.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3991.png,Normal/images/Normal-3991.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3991.png,Normal/masks/Normal-3991.png,True,False,299,299,L,166.6,a6aa0808fef1266f0869515b21ff6b32
Normal-3992.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3992.png,Normal/images/Normal-3992.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3992.png,Normal/masks/Normal-3992.png,True,False,299,299,L,166.04,8860af9cdb116f40390ff0a7cdee1154
Normal-3993.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3993.png,Normal/images/Normal-3993.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3993.png,Normal/masks/Normal-3993.png,True,False,299,299,L,104.02,f53c062f00c68f527ee4aaa077a6e2b2
Normal-3994.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3994.png,Normal/images/Normal-3994.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3994.png,Normal/masks/Normal-3994.png,True,False,299,299,L,123.14,a4cc976291c186c98c09dba351e55071
Normal-3995.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3995.png,Normal/images/Normal-3995.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3995.png,Normal/masks/Normal-3995.png,True,False,299,299,L,178.82,bce4b76320e50bade2d499347901431d
Normal-3996.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3996.png,Normal/images/Normal-3996.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3996.png,Normal/masks/Normal-3996.png,True,False,299,299,L,133.64,61c507d4e6f90344234af83bafc25396
Normal-3997.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3997.png,Normal/images/Normal-3997.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3997.png,Normal/masks/Normal-3997.png,True,False,299,299,L,109.43,8931566d78fcc146e67a67f25515a2f5
Normal-3998.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3998.png,Normal/images/Normal-3998.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3998.png,Normal/masks/Normal-3998.png,True,False,299,299,L,105.54,2e682dc3d215487fd7e5e2d1f0daf547
Normal-3999.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-3999.png,Normal/images/Normal-3999.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-3999.png,Normal/masks/Normal-3999.png,True,False,299,299,L,99.78,f620253300e0a3fd0a80bf8757c10f7f
Normal-4.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4.png,Normal/images/Normal-4.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4.png,Normal/masks/Normal-4.png,True,False,299,299,L,138.47,0bb5e02779d259ca36f25e2c1e46025b
Normal-40.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-40.png,Normal/images/Normal-40.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-40.png,Normal/masks/Normal-40.png,True,False,299,299,L,142.03,89572698250451e46198aedbf78e03cb
Normal-400.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-400.png,Normal/images/Normal-400.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-400.png,Normal/masks/Normal-400.png,True,False,299,299,L,141.04,ef88276f9bdcb067933a680431669bcf
Normal-4000.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4000.png,Normal/images/Normal-4000.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4000.png,Normal/masks/Normal-4000.png,True,False,299,299,L,116.14,87b1dc957e6dfe6d0ece131523e14eef
Normal-4001.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4001.png,Normal/images/Normal-4001.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4001.png,Normal/masks/Normal-4001.png,True,False,299,299,L,125.79,1ff58269acc43a42eef6b331e4823f45
Normal-4002.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4002.png,Normal/images/Normal-4002.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4002.png,Normal/masks/Normal-4002.png,True,False,299,299,L,95.89,33f8bc5106f3d89f8b0d299b2639445b
Normal-4003.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4003.png,Normal/images/Normal-4003.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4003.png,Normal/masks/Normal-4003.png,True,False,299,299,L,125.53,8076147b3793da130ad8031f0cfaefb1
Normal-4004.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4004.png,Normal/images/Normal-4004.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4004.png,Normal/masks/Normal-4004.png,True,False,299,299,L,114.07,1106fe511df36996919af4835a358a8e
Normal-4005.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4005.png,Normal/images/Normal-4005.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4005.png,Normal/masks/Normal-4005.png,True,False,299,299,L,90.48,870169206c157c3eca83ad0ba75c9b01
Normal-4006.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4006.png,Normal/images/Normal-4006.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4006.png,Normal/masks/Normal-4006.png,True,False,299,299,L,150.37,f209d8733e91d4b69597c2607b103cf2
Normal-4007.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4007.png,Normal/images/Normal-4007.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4007.png,Normal/masks/Normal-4007.png,True,False,299,299,L,110.75,48c6a56b6a7fa13b9136306dc5bc5992
Normal-4008.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4008.png,Normal/images/Normal-4008.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4008.png,Normal/masks/Normal-4008.png,True,False,299,299,L,111.73,544ee7687e1017afee227387bf1dfc7c
Normal-4009.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4009.png,Normal/images/Normal-4009.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4009.png,Normal/masks/Normal-4009.png,True,False,299,299,L,149.06,0b0d904a42ddad9cbadcdefe802927b6
Normal-401.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-401.png,Normal/images/Normal-401.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-401.png,Normal/masks/Normal-401.png,True,False,299,299,L,125.51,806b24da3b1b1a7527746c3b048e05ae
Normal-4010.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4010.png,Normal/images/Normal-4010.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4010.png,Normal/masks/Normal-4010.png,True,False,299,299,L,135.3,82d8af58c932063f98ab1fed05d7fa5c
Normal-4011.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4011.png,Normal/images/Normal-4011.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4011.png,Normal/masks/Normal-4011.png,True,False,299,299,L,105.43,1455e54b919a057a21225edc24d47f5b
Normal-4012.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4012.png,Normal/images/Normal-4012.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4012.png,Normal/masks/Normal-4012.png,True,False,299,299,L,173.47,1b59e76753a1dfe85e11fdf870b534c4
Normal-4013.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4013.png,Normal/images/Normal-4013.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4013.png,Normal/masks/Normal-4013.png,True,False,299,299,L,119.77,62f5963faf00bd17b59ed5492a11a9bc
Normal-4014.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4014.png,Normal/images/Normal-4014.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4014.png,Normal/masks/Normal-4014.png,True,False,299,299,L,169.56,8c0e5f4d5cbedfafede0932af51f2578
Normal-4015.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4015.png,Normal/images/Normal-4015.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4015.png,Normal/masks/Normal-4015.png,True,False,299,299,L,142.17,a207531c402fc1b5e425e42517d4a676
Normal-4016.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4016.png,Normal/images/Normal-4016.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4016.png,Normal/masks/Normal-4016.png,True,False,299,299,L,129.75,1be45417a176b2ddc8a7726801e4ee0d
Normal-4017.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4017.png,Normal/images/Normal-4017.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4017.png,Normal/masks/Normal-4017.png,True,False,299,299,L,105.25,71cefb841b08976a896afc9aa670dad4
Normal-4018.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4018.png,Normal/images/Normal-4018.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4018.png,Normal/masks/Normal-4018.png,True,False,299,299,L,132.53,36182af6e6b2ca024c6ebe1f29c7500b
Normal-4019.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4019.png,Normal/images/Normal-4019.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4019.png,Normal/masks/Normal-4019.png,True,False,299,299,L,143.55,27afec800ce5ff93358268adca2ce256
Normal-402.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-402.png,Normal/images/Normal-402.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-402.png,Normal/masks/Normal-402.png,True,False,299,299,L,121.42,20400c3c3439a5e204ff60464baff43d
Normal-4020.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4020.png,Normal/images/Normal-4020.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4020.png,Normal/masks/Normal-4020.png,True,False,299,299,L,119.41,20349410cad459cdf22c7a293fe4e55a
Normal-4021.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4021.png,Normal/images/Normal-4021.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4021.png,Normal/masks/Normal-4021.png,True,False,299,299,L,117.08,5c23784114626354dca1a5701876e93b
Normal-4022.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4022.png,Normal/images/Normal-4022.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4022.png,Normal/masks/Normal-4022.png,True,False,299,299,L,168.73,327851359970837d84d482e8546bb285
Normal-4023.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4023.png,Normal/images/Normal-4023.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4023.png,Normal/masks/Normal-4023.png,True,False,299,299,L,122.57,8f1b9397dfe59ae87a0b6131ba0b1592
Normal-4024.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4024.png,Normal/images/Normal-4024.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4024.png,Normal/masks/Normal-4024.png,True,False,299,299,L,136.55,6926a001a59126d6dfaa1f7fc9ad9cb4
Normal-4025.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4025.png,Normal/images/Normal-4025.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4025.png,Normal/masks/Normal-4025.png,True,False,299,299,L,114.03,dce5b25cb10bb67c71f0c708e2b21278
Normal-4026.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4026.png,Normal/images/Normal-4026.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4026.png,Normal/masks/Normal-4026.png,True,False,299,299,L,117.77,58d010018620d970b397b7f7fb97ae15
Normal-4027.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4027.png,Normal/images/Normal-4027.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4027.png,Normal/masks/Normal-4027.png,True,False,299,299,L,154.66,7c1248686b3d264062dcabf7fba5c504
Normal-4028.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4028.png,Normal/images/Normal-4028.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4028.png,Normal/masks/Normal-4028.png,True,False,299,299,L,154.92,ea4179e550cd96046ba65e27067035bb
Normal-4029.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4029.png,Normal/images/Normal-4029.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4029.png,Normal/masks/Normal-4029.png,True,False,299,299,L,162.05,a63fef72485046455d815127dab332f7
Normal-403.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-403.png,Normal/images/Normal-403.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-403.png,Normal/masks/Normal-403.png,True,False,299,299,L,130.09,646d4ed00b276ab69787645038113a1d
Normal-4030.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4030.png,Normal/images/Normal-4030.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4030.png,Normal/masks/Normal-4030.png,True,False,299,299,L,158.1,320de8a30ceb167f14e83bbee4d0eb18
Normal-4031.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4031.png,Normal/images/Normal-4031.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4031.png,Normal/masks/Normal-4031.png,True,False,299,299,L,157.29,00036e95d11e9bf3c7e504477557b375
Normal-4032.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4032.png,Normal/images/Normal-4032.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4032.png,Normal/masks/Normal-4032.png,True,False,299,299,L,115.71,642aff0bf6803aebb172614f0c0f7424
Normal-4033.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4033.png,Normal/images/Normal-4033.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4033.png,Normal/masks/Normal-4033.png,True,False,299,299,L,106.17,b9a87abe93a1937684b209689fcbed64
Normal-4034.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4034.png,Normal/images/Normal-4034.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4034.png,Normal/masks/Normal-4034.png,True,False,299,299,L,82.81,35590ac1d0748be2f562b8c00059c688
Normal-4035.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4035.png,Normal/images/Normal-4035.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4035.png,Normal/masks/Normal-4035.png,True,False,299,299,L,172.49,27848321f6eb3be5f90359bb48fe8a6e
Normal-4036.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4036.png,Normal/images/Normal-4036.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4036.png,Normal/masks/Normal-4036.png,True,False,299,299,L,116.77,ce46928f3ed18962fb7a4f3b54b380b4
Normal-4037.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4037.png,Normal/images/Normal-4037.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4037.png,Normal/masks/Normal-4037.png,True,False,299,299,L,103.9,81fd80ca52a36bb8655499f761523025
Normal-4038.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4038.png,Normal/images/Normal-4038.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4038.png,Normal/masks/Normal-4038.png,True,False,299,299,L,130.59,0195bd611b33c99f6b68dc207b8d6a32
Normal-4039.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4039.png,Normal/images/Normal-4039.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4039.png,Normal/masks/Normal-4039.png,True,False,299,299,L,112.03,80f58ea44f539bc80c58553acd992fd9
Normal-404.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-404.png,Normal/images/Normal-404.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-404.png,Normal/masks/Normal-404.png,True,False,299,299,L,140.08,50fce92a0fe39574080af8200dc99e81
Normal-4040.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4040.png,Normal/images/Normal-4040.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4040.png,Normal/masks/Normal-4040.png,True,False,299,299,L,119.41,892ce4dbf8889c1ab2bb53537205f783
Normal-4041.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4041.png,Normal/images/Normal-4041.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4041.png,Normal/masks/Normal-4041.png,True,False,299,299,L,106.71,dd4282cd24edff2657de9a39c8610b59
Normal-4042.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4042.png,Normal/images/Normal-4042.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4042.png,Normal/masks/Normal-4042.png,True,False,299,299,L,131.62,6ca05cb69a25679c03faeae3618e7edd
Normal-4043.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4043.png,Normal/images/Normal-4043.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4043.png,Normal/masks/Normal-4043.png,True,False,299,299,L,119.13,36a2cc290839aad080c00c71de7937a8
Normal-4044.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4044.png,Normal/images/Normal-4044.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4044.png,Normal/masks/Normal-4044.png,True,False,299,299,L,96.6,6f479267c421bfd5627ae324f75374df
Normal-4045.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4045.png,Normal/images/Normal-4045.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4045.png,Normal/masks/Normal-4045.png,True,False,299,299,L,153.9,69bdd2abdf62efd7d9861b38ffec46d8
Normal-4046.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4046.png,Normal/images/Normal-4046.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4046.png,Normal/masks/Normal-4046.png,True,False,299,299,L,125.43,965e5110536217367893a5fe0ffc4eae
Normal-4047.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4047.png,Normal/images/Normal-4047.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4047.png,Normal/masks/Normal-4047.png,True,False,299,299,L,174.86,d345207e4f80a69f0018be134d8d2284
Normal-4048.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4048.png,Normal/images/Normal-4048.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4048.png,Normal/masks/Normal-4048.png,True,False,299,299,L,169.07,2bea7e052d79c7b427b085dfaa002fe0
Normal-4049.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4049.png,Normal/images/Normal-4049.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4049.png,Normal/masks/Normal-4049.png,True,False,299,299,L,101.11,62c0f2ed32f2f7cb97716ffe9412b374
Normal-405.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-405.png,Normal/images/Normal-405.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-405.png,Normal/masks/Normal-405.png,True,False,299,299,L,116.49,c969e75200aa8f97c4c9873df1da9dfe
Normal-4050.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4050.png,Normal/images/Normal-4050.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4050.png,Normal/masks/Normal-4050.png,True,False,299,299,L,111.64,ffd6f950a73d5573e6cbced570ef3a04
Normal-4051.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4051.png,Normal/images/Normal-4051.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4051.png,Normal/masks/Normal-4051.png,True,False,299,299,L,165.01,1d48ccb725db4768a8b441bb79890dae
Normal-4052.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4052.png,Normal/images/Normal-4052.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4052.png,Normal/masks/Normal-4052.png,True,False,299,299,L,139.62,6f63ee7d9b251a86cdc61c7d2c72c85e
Normal-4053.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4053.png,Normal/images/Normal-4053.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4053.png,Normal/masks/Normal-4053.png,True,False,299,299,L,103.86,e29fcaa7b2c8b2370dd2c9572ec66f81
Normal-4054.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4054.png,Normal/images/Normal-4054.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4054.png,Normal/masks/Normal-4054.png,True,False,299,299,L,138.67,cdb723e466893c9ebfadb5a41d77f8d1
Normal-4055.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4055.png,Normal/images/Normal-4055.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4055.png,Normal/masks/Normal-4055.png,True,False,299,299,L,112.67,47e6ec5cfad2de4dfa56a9fb75022ee3
Normal-4056.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4056.png,Normal/images/Normal-4056.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4056.png,Normal/masks/Normal-4056.png,True,False,299,299,L,134.0,220f2a83db0562981efa0690ed8a4e0c
Normal-4057.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4057.png,Normal/images/Normal-4057.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4057.png,Normal/masks/Normal-4057.png,True,False,299,299,L,112.46,448bf18fc349fe94ae0742192e227c1b
Normal-4058.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4058.png,Normal/images/Normal-4058.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4058.png,Normal/masks/Normal-4058.png,True,False,299,299,L,98.43,e438c54a6589b57546cdb2ffdb67e111
Normal-4059.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4059.png,Normal/images/Normal-4059.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4059.png,Normal/masks/Normal-4059.png,True,False,299,299,L,98.44,73919d1486e5fe13a772dd17216fdf5b
Normal-406.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-406.png,Normal/images/Normal-406.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-406.png,Normal/masks/Normal-406.png,True,False,299,299,L,132.87,9129a49742cbcdab54913c6df012e495
Normal-4060.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4060.png,Normal/images/Normal-4060.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4060.png,Normal/masks/Normal-4060.png,True,False,299,299,L,141.64,7ba614714d0c46897099f73e4f6df844
Normal-4061.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4061.png,Normal/images/Normal-4061.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4061.png,Normal/masks/Normal-4061.png,True,False,299,299,L,98.43,5d285c161296abb2b89f5c4fc2afb5d3
Normal-4062.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4062.png,Normal/images/Normal-4062.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4062.png,Normal/masks/Normal-4062.png,True,False,299,299,L,165.51,c5b4a941d61c1e2d54b55b39a40e8fb9
Normal-4063.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4063.png,Normal/images/Normal-4063.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4063.png,Normal/masks/Normal-4063.png,True,False,299,299,L,110.12,3aabbeeabad202ae74c3947606e17909
Normal-4064.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4064.png,Normal/images/Normal-4064.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4064.png,Normal/masks/Normal-4064.png,True,False,299,299,L,167.26,c9e354ad6d7a0e6afa0cd17b3b5d779e
Normal-4065.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4065.png,Normal/images/Normal-4065.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4065.png,Normal/masks/Normal-4065.png,True,False,299,299,L,166.01,9ae7b4333b1ebdf66d0cf5aac6eb5ca4
Normal-4066.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4066.png,Normal/images/Normal-4066.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4066.png,Normal/masks/Normal-4066.png,True,False,299,299,L,115.52,658b105ffbaa8721d844a9e1e3920ffe
Normal-4067.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4067.png,Normal/images/Normal-4067.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4067.png,Normal/masks/Normal-4067.png,True,False,299,299,L,113.05,019f68c1f99b1ab3c19c278810a3d8d9
Normal-4068.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4068.png,Normal/images/Normal-4068.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4068.png,Normal/masks/Normal-4068.png,True,False,299,299,L,149.61,b088af70f798cf530a16f5770e214ad0
Normal-4069.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4069.png,Normal/images/Normal-4069.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4069.png,Normal/masks/Normal-4069.png,True,False,299,299,L,118.12,86d536dc33dad06895eb3d3c5b5e3fc5
Normal-407.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-407.png,Normal/images/Normal-407.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-407.png,Normal/masks/Normal-407.png,True,False,299,299,L,115.58,28ab9d5ab1e065b5bbd78e487bd9bda6
Normal-4070.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4070.png,Normal/images/Normal-4070.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4070.png,Normal/masks/Normal-4070.png,True,False,299,299,L,143.5,896756eab7bbf276bfde5ba2fb1c91f9
Normal-4071.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4071.png,Normal/images/Normal-4071.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4071.png,Normal/masks/Normal-4071.png,True,False,299,299,L,166.52,90bf0120f165378bdbfec1171a049d73
Normal-4072.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4072.png,Normal/images/Normal-4072.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4072.png,Normal/masks/Normal-4072.png,True,False,299,299,L,168.94,f797291658925ba426a0d1ec37558b6b
Normal-4073.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4073.png,Normal/images/Normal-4073.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4073.png,Normal/masks/Normal-4073.png,True,False,299,299,L,156.52,ae2dee28c0ae26ac6ffe130a1de60d60
Normal-4074.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4074.png,Normal/images/Normal-4074.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4074.png,Normal/masks/Normal-4074.png,True,False,299,299,L,108.28,f4fe943c569f9fc8571aa2a76150fba4
Normal-4075.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4075.png,Normal/images/Normal-4075.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4075.png,Normal/masks/Normal-4075.png,True,False,299,299,L,102.93,58a47641810f6c0590c2db39ca7bb421
Normal-4076.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4076.png,Normal/images/Normal-4076.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4076.png,Normal/masks/Normal-4076.png,True,False,299,299,L,150.6,2b92c8fa7ac32bfbfe5ceb7510015f9e
Normal-4077.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4077.png,Normal/images/Normal-4077.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4077.png,Normal/masks/Normal-4077.png,True,False,299,299,L,120.44,b277293c48c3d962640f978d4237219b
Normal-4078.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4078.png,Normal/images/Normal-4078.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4078.png,Normal/masks/Normal-4078.png,True,False,299,299,L,134.17,d7541e97ef5b022253c01112a882bfc3
Normal-4079.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4079.png,Normal/images/Normal-4079.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4079.png,Normal/masks/Normal-4079.png,True,False,299,299,L,109.57,64ad2af1277f5e05be3a0a9b6ce2f19a
Normal-408.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-408.png,Normal/images/Normal-408.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-408.png,Normal/masks/Normal-408.png,True,False,299,299,L,118.44,30fcc07833491000003a52203aab978c
Normal-4080.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4080.png,Normal/images/Normal-4080.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4080.png,Normal/masks/Normal-4080.png,True,False,299,299,L,110.0,de8ec36c752a7969a8e38ff33d986908
Normal-4081.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4081.png,Normal/images/Normal-4081.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4081.png,Normal/masks/Normal-4081.png,True,False,299,299,L,108.07,43dd06c8756e7b13948ca7240e3383e8
Normal-4082.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4082.png,Normal/images/Normal-4082.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4082.png,Normal/masks/Normal-4082.png,True,False,299,299,L,139.68,f38f1d4b3436e1873a838688dcf2e3a8
Normal-4083.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4083.png,Normal/images/Normal-4083.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4083.png,Normal/masks/Normal-4083.png,True,False,299,299,L,118.04,f7c12d57edfdcc5c7b5e4a010b0e66a5
Normal-4084.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4084.png,Normal/images/Normal-4084.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4084.png,Normal/masks/Normal-4084.png,True,False,299,299,L,115.31,4a6379c90b70922cdde66e54c69dcd3e
Normal-4085.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4085.png,Normal/images/Normal-4085.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4085.png,Normal/masks/Normal-4085.png,True,False,299,299,L,104.47,922e331cacaf6a5183f1688810dcf101
Normal-4086.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4086.png,Normal/images/Normal-4086.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4086.png,Normal/masks/Normal-4086.png,True,False,299,299,L,107.3,ab6ec0a136979094f08bd982a7759cbc
Normal-4087.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4087.png,Normal/images/Normal-4087.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4087.png,Normal/masks/Normal-4087.png,True,False,299,299,L,141.2,16fc22368fc9ecd5fa6c9f6fc19295ce
Normal-4088.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4088.png,Normal/images/Normal-4088.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4088.png,Normal/masks/Normal-4088.png,True,False,299,299,L,150.67,317d65ce8f53ca7a2aec6b5e785d5c7a
Normal-4089.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4089.png,Normal/images/Normal-4089.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4089.png,Normal/masks/Normal-4089.png,True,False,299,299,L,151.44,b5c909e24856f9daef1c7bb925cdb736
Normal-409.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-409.png,Normal/images/Normal-409.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-409.png,Normal/masks/Normal-409.png,True,False,299,299,L,108.35,723c2e93c2030c31624aca4249cde9ce
Normal-4090.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4090.png,Normal/images/Normal-4090.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4090.png,Normal/masks/Normal-4090.png,True,False,299,299,L,100.73,93d07583e4768d259e2efdd43af5c740
Normal-4091.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4091.png,Normal/images/Normal-4091.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4091.png,Normal/masks/Normal-4091.png,True,False,299,299,L,93.94,199eec90aaa9a24ef4b0672e019f52eb
Normal-4092.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4092.png,Normal/images/Normal-4092.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4092.png,Normal/masks/Normal-4092.png,True,False,299,299,L,106.3,fc4b04f48599b8293d1c66a437b50916
Normal-4093.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4093.png,Normal/images/Normal-4093.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4093.png,Normal/masks/Normal-4093.png,True,False,299,299,L,148.63,b3d3350137f577392c17d2007f22c720
Normal-4094.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4094.png,Normal/images/Normal-4094.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4094.png,Normal/masks/Normal-4094.png,True,False,299,299,L,105.75,bb19a729d2702449b62ed45ed80d7ffa
Normal-4095.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4095.png,Normal/images/Normal-4095.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4095.png,Normal/masks/Normal-4095.png,True,False,299,299,L,114.81,e71d456f2c246405a512f43446a89dd2
Normal-4096.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4096.png,Normal/images/Normal-4096.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4096.png,Normal/masks/Normal-4096.png,True,False,299,299,L,136.72,8dbf7d2f8795ea418ab51d0d0323db09
Normal-4097.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4097.png,Normal/images/Normal-4097.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4097.png,Normal/masks/Normal-4097.png,True,False,299,299,L,99.77,995f255d569fbace7a6da0380be9550f
Normal-4098.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4098.png,Normal/images/Normal-4098.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4098.png,Normal/masks/Normal-4098.png,True,False,299,299,L,122.59,94321dc4a232ccc3780caeaac77a36b3
Normal-4099.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4099.png,Normal/images/Normal-4099.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4099.png,Normal/masks/Normal-4099.png,True,False,299,299,L,116.93,2726bcb19419b3c8a0afcdc66d95b728
Normal-41.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-41.png,Normal/images/Normal-41.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-41.png,Normal/masks/Normal-41.png,True,False,299,299,L,119.92,093a78c636e9a080a838a451a0187351
Normal-410.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-410.png,Normal/images/Normal-410.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-410.png,Normal/masks/Normal-410.png,True,False,299,299,L,114.54,9664f9f5345c88b5846876700a76e9d1
Normal-4100.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4100.png,Normal/images/Normal-4100.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4100.png,Normal/masks/Normal-4100.png,True,False,299,299,L,111.72,844ec4b91b1e90ea1cbf6de3a2745809
Normal-4101.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4101.png,Normal/images/Normal-4101.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4101.png,Normal/masks/Normal-4101.png,True,False,299,299,L,143.63,f14576f48ae23f5d72fe75cb7082cc01
Normal-4102.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4102.png,Normal/images/Normal-4102.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4102.png,Normal/masks/Normal-4102.png,True,False,299,299,L,120.07,79f4998d3bd2bce50276e27ed86d4488
Normal-4103.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4103.png,Normal/images/Normal-4103.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4103.png,Normal/masks/Normal-4103.png,True,False,299,299,L,155.87,87b42bd0f4fbd31ac8117286255e8120
Normal-4104.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4104.png,Normal/images/Normal-4104.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4104.png,Normal/masks/Normal-4104.png,True,False,299,299,L,131.27,df54f85e9227127963597d971795f159
Normal-4105.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4105.png,Normal/images/Normal-4105.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4105.png,Normal/masks/Normal-4105.png,True,False,299,299,L,112.42,fa91dd5e57ccacbde16b8000671a2de7
Normal-4106.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4106.png,Normal/images/Normal-4106.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4106.png,Normal/masks/Normal-4106.png,True,False,299,299,L,157.15,b575a1dd58dd46411553105ba81f913b
Normal-4107.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4107.png,Normal/images/Normal-4107.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4107.png,Normal/masks/Normal-4107.png,True,False,299,299,L,124.9,801e66f8e0dffc1a951977ffd02349b3
Normal-4108.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4108.png,Normal/images/Normal-4108.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4108.png,Normal/masks/Normal-4108.png,True,False,299,299,L,126.84,7404da027186e8ba8ffab1022e40001d
Normal-4109.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4109.png,Normal/images/Normal-4109.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4109.png,Normal/masks/Normal-4109.png,True,False,299,299,L,99.83,d6be9c79d88737fb457fa5ec44a376c2
Normal-411.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-411.png,Normal/images/Normal-411.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-411.png,Normal/masks/Normal-411.png,True,False,299,299,L,121.52,3400aa055f1a0db1acb62e08751ad51d
Normal-4110.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4110.png,Normal/images/Normal-4110.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4110.png,Normal/masks/Normal-4110.png,True,False,299,299,L,111.21,4e4be4b97584bee0d128d4afc1ee88d2
Normal-4111.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4111.png,Normal/images/Normal-4111.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4111.png,Normal/masks/Normal-4111.png,True,False,299,299,L,139.6,dc8eadcfcb42cb5712d58e393071e011
Normal-4112.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4112.png,Normal/images/Normal-4112.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4112.png,Normal/masks/Normal-4112.png,True,False,299,299,L,110.7,825fccf2b426d70e3dd9fae062a8e8b1
Normal-4113.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4113.png,Normal/images/Normal-4113.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4113.png,Normal/masks/Normal-4113.png,True,False,299,299,L,106.88,15b8db25db6e9574d695bacacff45693
Normal-4114.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4114.png,Normal/images/Normal-4114.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4114.png,Normal/masks/Normal-4114.png,True,False,299,299,L,178.42,fdd1d3e68b5ac8e3c17c71104da2fb6d
Normal-4115.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4115.png,Normal/images/Normal-4115.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4115.png,Normal/masks/Normal-4115.png,True,False,299,299,L,136.03,d56c9abf20fd12e1f0f139e35af8ad43
Normal-4116.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4116.png,Normal/images/Normal-4116.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4116.png,Normal/masks/Normal-4116.png,True,False,299,299,L,115.45,bacaf983c789bfef501e759ef20f5458
Normal-4117.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4117.png,Normal/images/Normal-4117.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4117.png,Normal/masks/Normal-4117.png,True,False,299,299,L,88.27,0047d03fdb42a5061ab478cf8ba927b9
Normal-4118.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4118.png,Normal/images/Normal-4118.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4118.png,Normal/masks/Normal-4118.png,True,False,299,299,L,96.87,da9286c2e21a61665db51a142c87d7a1
Normal-4119.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4119.png,Normal/images/Normal-4119.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4119.png,Normal/masks/Normal-4119.png,True,False,299,299,L,176.34,ec488dd8c7e4710dba9e1de368c36ca8
Normal-412.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-412.png,Normal/images/Normal-412.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-412.png,Normal/masks/Normal-412.png,True,False,299,299,L,128.73,6a17839c5ff39dd263b1f5808ac6c82a
Normal-4120.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4120.png,Normal/images/Normal-4120.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4120.png,Normal/masks/Normal-4120.png,True,False,299,299,L,135.62,753db8d9c275a7791e2f51c9b0b82c51
Normal-4121.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4121.png,Normal/images/Normal-4121.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4121.png,Normal/masks/Normal-4121.png,True,False,299,299,L,170.15,7d8c262d133b4d32ab8d5228cd411cc7
Normal-4122.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4122.png,Normal/images/Normal-4122.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4122.png,Normal/masks/Normal-4122.png,True,False,299,299,L,120.02,39d7513e865565f41142ccbc110489d2
Normal-4123.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4123.png,Normal/images/Normal-4123.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4123.png,Normal/masks/Normal-4123.png,True,False,299,299,L,119.19,0fd97dd3ba2fe629d9a254fc70a36588
Normal-4124.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4124.png,Normal/images/Normal-4124.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4124.png,Normal/masks/Normal-4124.png,True,False,299,299,L,137.81,5ae791acd62c25bd4489573db291c1fd
Normal-4125.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4125.png,Normal/images/Normal-4125.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4125.png,Normal/masks/Normal-4125.png,True,False,299,299,L,118.87,c9f5557314479ab10965b6bde68d51d3
Normal-4126.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4126.png,Normal/images/Normal-4126.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4126.png,Normal/masks/Normal-4126.png,True,False,299,299,L,110.47,cc7eb40d835b14fc5c14378e46791ee8
Normal-4127.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4127.png,Normal/images/Normal-4127.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4127.png,Normal/masks/Normal-4127.png,True,False,299,299,L,113.67,df4781d2c65dd26d5f863b6e4aba1f81
Normal-4128.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4128.png,Normal/images/Normal-4128.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4128.png,Normal/masks/Normal-4128.png,True,False,299,299,L,128.39,9701f109c9ff193b45e596f6187296ba
Normal-4129.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4129.png,Normal/images/Normal-4129.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4129.png,Normal/masks/Normal-4129.png,True,False,299,299,L,127.72,961fa01ef32039690b092a256e19ee67
Normal-413.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-413.png,Normal/images/Normal-413.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-413.png,Normal/masks/Normal-413.png,True,False,299,299,L,130.38,67dc96283444d092c6d75c20590ffbd9
Normal-4130.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4130.png,Normal/images/Normal-4130.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4130.png,Normal/masks/Normal-4130.png,True,False,299,299,L,110.45,19056ba8f214cddc1b57d59eebd2b693
Normal-4131.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4131.png,Normal/images/Normal-4131.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4131.png,Normal/masks/Normal-4131.png,True,False,299,299,L,135.07,2475d6545c3f2f0ccd629e4f28495eb8
Normal-4132.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4132.png,Normal/images/Normal-4132.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4132.png,Normal/masks/Normal-4132.png,True,False,299,299,L,165.65,8a593644b4f96871220397347bcc4df9
Normal-4133.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4133.png,Normal/images/Normal-4133.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4133.png,Normal/masks/Normal-4133.png,True,False,299,299,L,189.8,ba626b25507c1095e69998be79530c35
Normal-4134.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4134.png,Normal/images/Normal-4134.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4134.png,Normal/masks/Normal-4134.png,True,False,299,299,L,131.3,583f12890980df26553d1e4be349e0b8
Normal-4135.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4135.png,Normal/images/Normal-4135.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4135.png,Normal/masks/Normal-4135.png,True,False,299,299,L,89.33,18d931238d2f623859146b2bd64f299e
Normal-4136.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4136.png,Normal/images/Normal-4136.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4136.png,Normal/masks/Normal-4136.png,True,False,299,299,L,147.72,32936844196d91d0389fd216f2c90704
Normal-4137.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4137.png,Normal/images/Normal-4137.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4137.png,Normal/masks/Normal-4137.png,True,False,299,299,L,118.65,2b866a1c24e8a6ae429c8c475496b961
Normal-4138.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4138.png,Normal/images/Normal-4138.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4138.png,Normal/masks/Normal-4138.png,True,False,299,299,L,108.48,9d660327d2ca7464767513b743ef2933
Normal-4139.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4139.png,Normal/images/Normal-4139.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4139.png,Normal/masks/Normal-4139.png,True,False,299,299,L,95.56,eb5c49015ede1de43515133f65aaeebe
Normal-414.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-414.png,Normal/images/Normal-414.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-414.png,Normal/masks/Normal-414.png,True,False,299,299,L,114.91,bfd6778b193531f254f24c9123d9c68a
Normal-4140.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4140.png,Normal/images/Normal-4140.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4140.png,Normal/masks/Normal-4140.png,True,False,299,299,L,138.96,98681faf2fe6ccdfbc4a08d98e16521d
Normal-4141.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4141.png,Normal/images/Normal-4141.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4141.png,Normal/masks/Normal-4141.png,True,False,299,299,L,100.72,a04499c5fbd0e9a701bc8858ab6aad67
Normal-4142.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4142.png,Normal/images/Normal-4142.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4142.png,Normal/masks/Normal-4142.png,True,False,299,299,L,120.17,1a543d65dd70aa285f3cce1e4a2adf3b
Normal-4143.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4143.png,Normal/images/Normal-4143.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4143.png,Normal/masks/Normal-4143.png,True,False,299,299,L,158.71,6b573c85c47f6e5db2be0776052b8947
Normal-4144.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4144.png,Normal/images/Normal-4144.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4144.png,Normal/masks/Normal-4144.png,True,False,299,299,L,162.77,58907c1ad8eb1012406701880405bc40
Normal-4145.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4145.png,Normal/images/Normal-4145.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4145.png,Normal/masks/Normal-4145.png,True,False,299,299,L,167.1,2dc6bdfc34802134d21c2f13f61fbf83
Normal-4146.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4146.png,Normal/images/Normal-4146.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4146.png,Normal/masks/Normal-4146.png,True,False,299,299,L,123.62,7046a43d1e37bf8847e1355218b3d7cc
Normal-4147.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4147.png,Normal/images/Normal-4147.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4147.png,Normal/masks/Normal-4147.png,True,False,299,299,L,112.36,647f681294fece4fb1a6f1a0cd6eab56
Normal-4148.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4148.png,Normal/images/Normal-4148.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4148.png,Normal/masks/Normal-4148.png,True,False,299,299,L,178.94,bddcf5740fa3a91c1783d9db37a26105
Normal-4149.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4149.png,Normal/images/Normal-4149.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4149.png,Normal/masks/Normal-4149.png,True,False,299,299,L,122.73,a625d8082df2352f77062133f9c58859
Normal-415.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-415.png,Normal/images/Normal-415.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-415.png,Normal/masks/Normal-415.png,True,False,299,299,L,122.57,d44843886e45cfaa2cd423b2af102058
Normal-4150.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4150.png,Normal/images/Normal-4150.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4150.png,Normal/masks/Normal-4150.png,True,False,299,299,L,104.08,d22067e88cc137a8f5b2f9a37398b632
Normal-4151.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4151.png,Normal/images/Normal-4151.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4151.png,Normal/masks/Normal-4151.png,True,False,299,299,L,168.37,fb4b685a2b57fa400cff3ad266c81e1c
Normal-4152.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4152.png,Normal/images/Normal-4152.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4152.png,Normal/masks/Normal-4152.png,True,False,299,299,L,122.24,dd59fcafc0f531d20d9851bdcbb82b4b
Normal-4153.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4153.png,Normal/images/Normal-4153.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4153.png,Normal/masks/Normal-4153.png,True,False,299,299,L,161.72,541313e0bad18cf49aa3ea0fe55065b5
Normal-4154.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4154.png,Normal/images/Normal-4154.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4154.png,Normal/masks/Normal-4154.png,True,False,299,299,L,120.07,624ce9e08473dc3452f6e39d20e13a6d
Normal-4155.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4155.png,Normal/images/Normal-4155.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4155.png,Normal/masks/Normal-4155.png,True,False,299,299,L,101.97,50d257fc68a5fcf8724b9b1e051cea67
Normal-4156.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4156.png,Normal/images/Normal-4156.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4156.png,Normal/masks/Normal-4156.png,True,False,299,299,L,160.88,6b73aafb3dae14a6ad9053c0954bdc6f
Normal-4157.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4157.png,Normal/images/Normal-4157.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4157.png,Normal/masks/Normal-4157.png,True,False,299,299,L,125.85,852b6c8ea12eee77d6d1feedd802aa70
Normal-4158.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4158.png,Normal/images/Normal-4158.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4158.png,Normal/masks/Normal-4158.png,True,False,299,299,L,112.81,b4e01cde2707e1beaba54302f78abbf1
Normal-4159.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4159.png,Normal/images/Normal-4159.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4159.png,Normal/masks/Normal-4159.png,True,False,299,299,L,130.19,c83626589b8b156da61129faded27210
Normal-416.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-416.png,Normal/images/Normal-416.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-416.png,Normal/masks/Normal-416.png,True,False,299,299,L,130.91,3d82c6b7cb66ca01ac5413d44b380808
Normal-4160.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4160.png,Normal/images/Normal-4160.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4160.png,Normal/masks/Normal-4160.png,True,False,299,299,L,128.15,a403120ebddd9d8a343ba6e6881640c1
Normal-4161.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4161.png,Normal/images/Normal-4161.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4161.png,Normal/masks/Normal-4161.png,True,False,299,299,L,161.25,be6ea7999bb652aa675bc9a562509880
Normal-4162.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4162.png,Normal/images/Normal-4162.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4162.png,Normal/masks/Normal-4162.png,True,False,299,299,L,117.55,9d42426bc4a0abb00e91748861251b9f
Normal-4163.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4163.png,Normal/images/Normal-4163.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4163.png,Normal/masks/Normal-4163.png,True,False,299,299,L,124.56,b98878eff8ed0f67f8ba41520714f8e8
Normal-4164.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4164.png,Normal/images/Normal-4164.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4164.png,Normal/masks/Normal-4164.png,True,False,299,299,L,131.72,f584a9adbb6966f54e471895077c46a9
Normal-4165.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4165.png,Normal/images/Normal-4165.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4165.png,Normal/masks/Normal-4165.png,True,False,299,299,L,158.25,9c44b0bdce674bf72a0d447b87dc3b4e
Normal-4166.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4166.png,Normal/images/Normal-4166.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4166.png,Normal/masks/Normal-4166.png,True,False,299,299,L,153.71,808db68115f34aeb68747f8ad89f9cbb
Normal-4167.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4167.png,Normal/images/Normal-4167.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4167.png,Normal/masks/Normal-4167.png,True,False,299,299,L,175.7,13f0375066efb50f0e03ae74fe662406
Normal-4168.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4168.png,Normal/images/Normal-4168.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4168.png,Normal/masks/Normal-4168.png,True,False,299,299,L,125.78,484f19ab08c3553952ae715fd813a853
Normal-4169.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4169.png,Normal/images/Normal-4169.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4169.png,Normal/masks/Normal-4169.png,True,False,299,299,L,119.49,6339a3e3250e1fcad276f076e2d607f8
Normal-417.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-417.png,Normal/images/Normal-417.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-417.png,Normal/masks/Normal-417.png,True,False,299,299,L,112.87,e9a13095032cc03c9443a7af1cc0a288
Normal-4170.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4170.png,Normal/images/Normal-4170.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4170.png,Normal/masks/Normal-4170.png,True,False,299,299,L,94.02,934710454e81df0025bceea137a08d1b
Normal-4171.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4171.png,Normal/images/Normal-4171.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4171.png,Normal/masks/Normal-4171.png,True,False,299,299,L,153.61,2cf9a01c630bf4b43bef8e76434cc015
Normal-4172.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4172.png,Normal/images/Normal-4172.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4172.png,Normal/masks/Normal-4172.png,True,False,299,299,L,121.82,ea7bdce07a8c6d82dd8304fea5c6c58a
Normal-4173.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4173.png,Normal/images/Normal-4173.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4173.png,Normal/masks/Normal-4173.png,True,False,299,299,L,102.7,932e93f6453bf2b1f91216a09e8e6839
Normal-4174.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4174.png,Normal/images/Normal-4174.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4174.png,Normal/masks/Normal-4174.png,True,False,299,299,L,129.93,2bb4d327118bf01531859a1785954bbc
Normal-4175.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4175.png,Normal/images/Normal-4175.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4175.png,Normal/masks/Normal-4175.png,True,False,299,299,L,112.07,1eac20007517e7789f10a1e3c9e6a2b1
Normal-4176.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4176.png,Normal/images/Normal-4176.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4176.png,Normal/masks/Normal-4176.png,True,False,299,299,L,113.88,b48186ea1e514cbc05dcac6c05ffe648
Normal-4177.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4177.png,Normal/images/Normal-4177.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4177.png,Normal/masks/Normal-4177.png,True,False,299,299,L,129.12,75f0d4528279677158b855fa51136848
Normal-4178.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4178.png,Normal/images/Normal-4178.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4178.png,Normal/masks/Normal-4178.png,True,False,299,299,L,158.62,716c64119098004a88190af2fde4e0ab
Normal-4179.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4179.png,Normal/images/Normal-4179.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4179.png,Normal/masks/Normal-4179.png,True,False,299,299,L,140.94,a2683b7c72d144ce1c1d976ded87bd8c
Normal-418.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-418.png,Normal/images/Normal-418.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-418.png,Normal/masks/Normal-418.png,True,False,299,299,L,135.75,ba42b4935f1b393b90f30088a4be6599
Normal-4180.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4180.png,Normal/images/Normal-4180.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4180.png,Normal/masks/Normal-4180.png,True,False,299,299,L,121.19,e64a45a77f487a8c52ca070ab4163333
Normal-4181.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4181.png,Normal/images/Normal-4181.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4181.png,Normal/masks/Normal-4181.png,True,False,299,299,L,146.98,908412decb298362f847360a998a984d
Normal-4182.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4182.png,Normal/images/Normal-4182.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4182.png,Normal/masks/Normal-4182.png,True,False,299,299,L,144.79,95fb7b2ef131904a5b834604a4586b38
Normal-4183.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4183.png,Normal/images/Normal-4183.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4183.png,Normal/masks/Normal-4183.png,True,False,299,299,L,111.67,d4c6f8c66fb53ef495f4532d48c4fd3d
Normal-4184.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4184.png,Normal/images/Normal-4184.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4184.png,Normal/masks/Normal-4184.png,True,False,299,299,L,120.51,ae7d659bb45cd47e2a6cf230d8f0f88f
Normal-4185.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4185.png,Normal/images/Normal-4185.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4185.png,Normal/masks/Normal-4185.png,True,False,299,299,L,121.98,b05c91ba41ebaf25b06e4ceb0ae61394
Normal-4186.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4186.png,Normal/images/Normal-4186.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4186.png,Normal/masks/Normal-4186.png,True,False,299,299,L,109.27,6782cb1407f6524f1f87ccd1340ed799
Normal-4187.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4187.png,Normal/images/Normal-4187.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4187.png,Normal/masks/Normal-4187.png,True,False,299,299,L,106.71,c093bf40e042a196b6ec2569495fa173
Normal-4188.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4188.png,Normal/images/Normal-4188.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4188.png,Normal/masks/Normal-4188.png,True,False,299,299,L,135.4,947ba757195e70e670d1591c22554f52
Normal-4189.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4189.png,Normal/images/Normal-4189.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4189.png,Normal/masks/Normal-4189.png,True,False,299,299,L,118.82,44eb4518f111f6ee3597417f857ea8e2
Normal-419.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-419.png,Normal/images/Normal-419.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-419.png,Normal/masks/Normal-419.png,True,False,299,299,L,105.51,4095032238407b127a2b257383e98209
Normal-4190.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4190.png,Normal/images/Normal-4190.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4190.png,Normal/masks/Normal-4190.png,True,False,299,299,L,103.65,40dcd047644e2a5ed7a15275e3f83946
Normal-4191.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4191.png,Normal/images/Normal-4191.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4191.png,Normal/masks/Normal-4191.png,True,False,299,299,L,150.95,d6fe11440cf36cbf474158923a35cd4d
Normal-4192.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4192.png,Normal/images/Normal-4192.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4192.png,Normal/masks/Normal-4192.png,True,False,299,299,L,100.04,818635f836c3f9e9953eb2d11b72e80f
Normal-4193.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4193.png,Normal/images/Normal-4193.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4193.png,Normal/masks/Normal-4193.png,True,False,299,299,L,153.35,54a62254eddb44ee2b2d6bf520894a69
Normal-4194.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4194.png,Normal/images/Normal-4194.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4194.png,Normal/masks/Normal-4194.png,True,False,299,299,L,160.3,5bc8fff9f4196f9dddfd4cb02532e80b
Normal-4195.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4195.png,Normal/images/Normal-4195.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4195.png,Normal/masks/Normal-4195.png,True,False,299,299,L,151.62,df399d4268a419ca2d830221c30495ac
Normal-4196.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4196.png,Normal/images/Normal-4196.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4196.png,Normal/masks/Normal-4196.png,True,False,299,299,L,120.57,f35a6d205759c9150ea35ea8ae2d6a1e
Normal-4197.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4197.png,Normal/images/Normal-4197.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4197.png,Normal/masks/Normal-4197.png,True,False,299,299,L,171.72,33cb88e61bbf5f9b77617865a8004d8a
Normal-4198.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4198.png,Normal/images/Normal-4198.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4198.png,Normal/masks/Normal-4198.png,True,False,299,299,L,139.74,dabd3d9a9c7c0724145af2c9a78e3238
Normal-4199.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4199.png,Normal/images/Normal-4199.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4199.png,Normal/masks/Normal-4199.png,True,False,299,299,L,119.87,bb1d7b1da3aae28f51e0fe4abb4057e1
Normal-42.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-42.png,Normal/images/Normal-42.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-42.png,Normal/masks/Normal-42.png,True,False,299,299,L,165.34,223751f21ea859ec382766a977d1c0c9
Normal-420.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-420.png,Normal/images/Normal-420.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-420.png,Normal/masks/Normal-420.png,True,False,299,299,L,135.05,26395a1da7b997ef6480060a02f9d1b1
Normal-4200.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4200.png,Normal/images/Normal-4200.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4200.png,Normal/masks/Normal-4200.png,True,False,299,299,L,149.03,725f9feb96b936aa7d4386afa7d68876
Normal-4201.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4201.png,Normal/images/Normal-4201.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4201.png,Normal/masks/Normal-4201.png,True,False,299,299,L,152.14,3324cdfabb54f090b3af3948e2e8a7e0
Normal-4202.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4202.png,Normal/images/Normal-4202.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4202.png,Normal/masks/Normal-4202.png,True,False,299,299,L,172.11,de46eba5c621c5baf52acddb0d0d501f
Normal-4203.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4203.png,Normal/images/Normal-4203.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4203.png,Normal/masks/Normal-4203.png,True,False,299,299,L,114.94,f2fbfb33d7908d9d1d31f0c3f8bb0e92
Normal-4204.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4204.png,Normal/images/Normal-4204.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4204.png,Normal/masks/Normal-4204.png,True,False,299,299,L,125.52,d58036eeb5491ed90d642fa7961b0f4c
Normal-4205.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4205.png,Normal/images/Normal-4205.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4205.png,Normal/masks/Normal-4205.png,True,False,299,299,L,143.15,1f6e3f9abcb5f3e077684ca5dee53a5f
Normal-4206.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4206.png,Normal/images/Normal-4206.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4206.png,Normal/masks/Normal-4206.png,True,False,299,299,L,51.59,184efd7dfe5bda2240959c280fc841b0
Normal-4207.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4207.png,Normal/images/Normal-4207.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4207.png,Normal/masks/Normal-4207.png,True,False,299,299,L,112.18,1870076782c1f8dce84f9697f9b362c4
Normal-4208.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4208.png,Normal/images/Normal-4208.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4208.png,Normal/masks/Normal-4208.png,True,False,299,299,L,151.96,3c4bc98c4cd029d841ceb9c2bf34502f
Normal-4209.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4209.png,Normal/images/Normal-4209.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4209.png,Normal/masks/Normal-4209.png,True,False,299,299,L,171.09,076990e1437c62160bed71343f27ab83
Normal-421.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-421.png,Normal/images/Normal-421.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-421.png,Normal/masks/Normal-421.png,True,False,299,299,L,118.6,ee0257c0c79fc29c9736c1f6ede23199
Normal-4210.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4210.png,Normal/images/Normal-4210.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4210.png,Normal/masks/Normal-4210.png,True,False,299,299,L,115.95,1a98933590f9e58bc4f7a056cbe51e1e
Normal-4211.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4211.png,Normal/images/Normal-4211.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4211.png,Normal/masks/Normal-4211.png,True,False,299,299,L,161.14,8d8984ef6719bc8a9d644d8fed07e97f
Normal-4212.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4212.png,Normal/images/Normal-4212.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4212.png,Normal/masks/Normal-4212.png,True,False,299,299,L,171.88,a4abc454b05e3aac4c55611ac2c562f9
Normal-4213.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4213.png,Normal/images/Normal-4213.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4213.png,Normal/masks/Normal-4213.png,True,False,299,299,L,26.44,ddf7d602f4add3fa129156488b9acef0
Normal-4214.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4214.png,Normal/images/Normal-4214.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4214.png,Normal/masks/Normal-4214.png,True,False,299,299,L,146.09,5aacdab36963b9dd8d5cda4671c622da
Normal-4215.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4215.png,Normal/images/Normal-4215.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4215.png,Normal/masks/Normal-4215.png,True,False,299,299,L,105.35,2c8ea0cfbb999613d7e197f80fc02bda
Normal-4216.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4216.png,Normal/images/Normal-4216.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4216.png,Normal/masks/Normal-4216.png,True,False,299,299,L,111.55,077792e00ec258eb9ae73410130225ca
Normal-4217.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4217.png,Normal/images/Normal-4217.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4217.png,Normal/masks/Normal-4217.png,True,False,299,299,L,111.0,12de22d930bc740157563ac867854fb9
Normal-4218.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4218.png,Normal/images/Normal-4218.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4218.png,Normal/masks/Normal-4218.png,True,False,299,299,L,97.64,d41c19f802afe0717f434c85316c0d43
Normal-4219.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4219.png,Normal/images/Normal-4219.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4219.png,Normal/masks/Normal-4219.png,True,False,299,299,L,113.0,121e3860d3552066fe0d982c4246afb7
Normal-422.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-422.png,Normal/images/Normal-422.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-422.png,Normal/masks/Normal-422.png,True,False,299,299,L,134.41,9f0465d2ea0494f08695b7f11d0de47a
Normal-4220.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4220.png,Normal/images/Normal-4220.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4220.png,Normal/masks/Normal-4220.png,True,False,299,299,L,103.27,04ce62d2dfd62f0212c5db6e101b7a6d
Normal-4221.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4221.png,Normal/images/Normal-4221.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4221.png,Normal/masks/Normal-4221.png,True,False,299,299,L,181.45,ac56bc848bfa94882749b03bd1883678
Normal-4222.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4222.png,Normal/images/Normal-4222.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4222.png,Normal/masks/Normal-4222.png,True,False,299,299,L,136.29,5a4aa8e66283980a23292ea2fa95cb2e
Normal-4223.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4223.png,Normal/images/Normal-4223.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4223.png,Normal/masks/Normal-4223.png,True,False,299,299,L,106.27,6f0c1fbc431a1c6c42312ee7ecfbcb7d
Normal-4224.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4224.png,Normal/images/Normal-4224.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4224.png,Normal/masks/Normal-4224.png,True,False,299,299,L,132.78,13b8c22d90c0cb82e86c9ac80d6488d8
Normal-4225.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4225.png,Normal/images/Normal-4225.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4225.png,Normal/masks/Normal-4225.png,True,False,299,299,L,127.04,d010c87a739b9a7bd57b2d8ce0f35f52
Normal-4226.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4226.png,Normal/images/Normal-4226.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4226.png,Normal/masks/Normal-4226.png,True,False,299,299,L,165.89,f3c455254decf6e7d89fd89332c005e8
Normal-4227.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4227.png,Normal/images/Normal-4227.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4227.png,Normal/masks/Normal-4227.png,True,False,299,299,L,128.04,167c6e3028639a402c800951fc004da0
Normal-4228.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4228.png,Normal/images/Normal-4228.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4228.png,Normal/masks/Normal-4228.png,True,False,299,299,L,121.59,5e0f9faf1258bb6ef2805a9fce14400a
Normal-4229.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4229.png,Normal/images/Normal-4229.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4229.png,Normal/masks/Normal-4229.png,True,False,299,299,L,128.41,ca9b8cf70603d5dea83ac6515f75bb6d
Normal-423.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-423.png,Normal/images/Normal-423.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-423.png,Normal/masks/Normal-423.png,True,False,299,299,L,121.91,5babc9550e33156a4df6015ff51abd95
Normal-4230.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4230.png,Normal/images/Normal-4230.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4230.png,Normal/masks/Normal-4230.png,True,False,299,299,L,132.31,783eff3c254dc8644baef0b71234a6d9
Normal-4231.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4231.png,Normal/images/Normal-4231.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4231.png,Normal/masks/Normal-4231.png,True,False,299,299,L,138.22,0c003356960d2c4b08774d410a33b562
Normal-4232.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4232.png,Normal/images/Normal-4232.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4232.png,Normal/masks/Normal-4232.png,True,False,299,299,L,113.03,1db66f6e9412500453c0ce6a38715a29
Normal-4233.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4233.png,Normal/images/Normal-4233.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4233.png,Normal/masks/Normal-4233.png,True,False,299,299,L,112.91,11ffe6d23e4bd178422216f73bb1fe48
Normal-4234.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4234.png,Normal/images/Normal-4234.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4234.png,Normal/masks/Normal-4234.png,True,False,299,299,L,130.67,c23730dd927d5b2a8108033d543f0dbc
Normal-4235.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4235.png,Normal/images/Normal-4235.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4235.png,Normal/masks/Normal-4235.png,True,False,299,299,L,107.26,8146f68bfc32ed10c04c55898ecf5bf8
Normal-4236.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4236.png,Normal/images/Normal-4236.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4236.png,Normal/masks/Normal-4236.png,True,False,299,299,L,116.32,21d7e1c72d71f4e03b7539f17b409db0
Normal-4237.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4237.png,Normal/images/Normal-4237.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4237.png,Normal/masks/Normal-4237.png,True,False,299,299,L,105.97,716b6d610e51372fbaed5fa31fe98055
Normal-4238.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4238.png,Normal/images/Normal-4238.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4238.png,Normal/masks/Normal-4238.png,True,False,299,299,L,110.67,a978887ed6cc968a17e67be59797f265
Normal-4239.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4239.png,Normal/images/Normal-4239.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4239.png,Normal/masks/Normal-4239.png,True,False,299,299,L,93.32,d936fd9bfc10dadad588810dfe7d9670
Normal-424.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-424.png,Normal/images/Normal-424.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-424.png,Normal/masks/Normal-424.png,True,False,299,299,L,130.85,ee21b25f14bf14c7ec0901f06621c932
Normal-4240.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4240.png,Normal/images/Normal-4240.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4240.png,Normal/masks/Normal-4240.png,True,False,299,299,L,123.09,4fbb9eed3e678c092420c8e99a18fc53
Normal-4241.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4241.png,Normal/images/Normal-4241.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4241.png,Normal/masks/Normal-4241.png,True,False,299,299,L,107.19,f2f510b2ef0105cb990a37208e37871c
Normal-4242.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4242.png,Normal/images/Normal-4242.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4242.png,Normal/masks/Normal-4242.png,True,False,299,299,L,120.66,cf4c5e1550557609c0fa6a643a5402f2
Normal-4243.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4243.png,Normal/images/Normal-4243.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4243.png,Normal/masks/Normal-4243.png,True,False,299,299,L,131.38,fc34f0adb59f4dd5a990153e9f604ba6
Normal-4244.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4244.png,Normal/images/Normal-4244.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4244.png,Normal/masks/Normal-4244.png,True,False,299,299,L,139.64,8c874ddcddb8c82dcb32344508323955
Normal-4245.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4245.png,Normal/images/Normal-4245.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4245.png,Normal/masks/Normal-4245.png,True,False,299,299,L,173.6,f1d70e4992dfb832ddd958060fc253a1
Normal-4246.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4246.png,Normal/images/Normal-4246.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4246.png,Normal/masks/Normal-4246.png,True,False,299,299,L,157.67,a4b76415cccd169f4721ea64be285e4e
Normal-4247.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4247.png,Normal/images/Normal-4247.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4247.png,Normal/masks/Normal-4247.png,True,False,299,299,L,99.9,dabc4adef58e06bd1b12ad82d5c87c3e
Normal-4248.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4248.png,Normal/images/Normal-4248.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4248.png,Normal/masks/Normal-4248.png,True,False,299,299,L,158.14,984183f679993d026ae70f9cc59e33b0
Normal-4249.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4249.png,Normal/images/Normal-4249.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4249.png,Normal/masks/Normal-4249.png,True,False,299,299,L,105.29,4caceacd77f6c58f03a4d2080115ae57
Normal-425.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-425.png,Normal/images/Normal-425.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-425.png,Normal/masks/Normal-425.png,True,False,299,299,L,114.9,abe9194b3df61921a5640b3709232018
Normal-4250.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4250.png,Normal/images/Normal-4250.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4250.png,Normal/masks/Normal-4250.png,True,False,299,299,L,116.27,c8590a325c41242c3e28e49c3ca4779c
Normal-4251.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4251.png,Normal/images/Normal-4251.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4251.png,Normal/masks/Normal-4251.png,True,False,299,299,L,118.51,b436950be87edfe5408f0ae541927313
Normal-4252.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4252.png,Normal/images/Normal-4252.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4252.png,Normal/masks/Normal-4252.png,True,False,299,299,L,146.8,8ba17e14ee11954dc68b6eefc959e584
Normal-4253.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4253.png,Normal/images/Normal-4253.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4253.png,Normal/masks/Normal-4253.png,True,False,299,299,L,110.09,df6e27a9e57e949596c5b911a628c36c
Normal-4254.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4254.png,Normal/images/Normal-4254.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4254.png,Normal/masks/Normal-4254.png,True,False,299,299,L,105.12,871a51eebfd4943cf4c1c719a7cc7c2b
Normal-4255.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4255.png,Normal/images/Normal-4255.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4255.png,Normal/masks/Normal-4255.png,True,False,299,299,L,108.48,7163e6ba4eba898af7d5be20188e1ccb
Normal-4256.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4256.png,Normal/images/Normal-4256.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4256.png,Normal/masks/Normal-4256.png,True,False,299,299,L,120.39,65707bc270b077ec34236c59f9fdfc19
Normal-4257.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4257.png,Normal/images/Normal-4257.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4257.png,Normal/masks/Normal-4257.png,True,False,299,299,L,127.88,82942779c8d824fff2a1e5250e992066
Normal-4258.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4258.png,Normal/images/Normal-4258.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4258.png,Normal/masks/Normal-4258.png,True,False,299,299,L,121.06,eafb10295a28d5859bc3e52a20208ac1
Normal-4259.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4259.png,Normal/images/Normal-4259.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4259.png,Normal/masks/Normal-4259.png,True,False,299,299,L,103.35,b5323edb3c30e60d038d48b898234911
Normal-426.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-426.png,Normal/images/Normal-426.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-426.png,Normal/masks/Normal-426.png,True,False,299,299,L,127.33,051ae77c8a6c164f75d0e43377d7e5e0
Normal-4260.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4260.png,Normal/images/Normal-4260.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4260.png,Normal/masks/Normal-4260.png,True,False,299,299,L,144.99,60fde917ce414305fd05a02d25088072
Normal-4261.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4261.png,Normal/images/Normal-4261.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4261.png,Normal/masks/Normal-4261.png,True,False,299,299,L,145.88,b713d21f28853d289c612baae16166ed
Normal-4262.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4262.png,Normal/images/Normal-4262.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4262.png,Normal/masks/Normal-4262.png,True,False,299,299,L,125.34,35c447894acc9a40045449af6a305819
Normal-4263.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4263.png,Normal/images/Normal-4263.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4263.png,Normal/masks/Normal-4263.png,True,False,299,299,L,141.68,5e4bb2f76263a677feaf7a66a61cbaf4
Normal-4264.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4264.png,Normal/images/Normal-4264.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4264.png,Normal/masks/Normal-4264.png,True,False,299,299,L,166.44,03caf16277f80111381d799d725a1848
Normal-4265.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4265.png,Normal/images/Normal-4265.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4265.png,Normal/masks/Normal-4265.png,True,False,299,299,L,161.34,68e7f8d50017501aa0e4807c27c9d6c3
Normal-4266.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4266.png,Normal/images/Normal-4266.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4266.png,Normal/masks/Normal-4266.png,True,False,299,299,L,153.65,8367f3d774971e71b24968b878f6a671
Normal-4267.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4267.png,Normal/images/Normal-4267.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4267.png,Normal/masks/Normal-4267.png,True,False,299,299,L,144.9,f4a7f4c597d2ac346e5494c6c16d9f28
Normal-4268.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4268.png,Normal/images/Normal-4268.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4268.png,Normal/masks/Normal-4268.png,True,False,299,299,L,171.35,01e32426fe1dba2de95e84c33b49bd2d
Normal-4269.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4269.png,Normal/images/Normal-4269.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4269.png,Normal/masks/Normal-4269.png,True,False,299,299,L,123.0,a97e8369a602ee124412b26fb87bb5f0
Normal-427.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-427.png,Normal/images/Normal-427.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-427.png,Normal/masks/Normal-427.png,True,False,299,299,L,125.32,dfba752719a3c31e71e10ac280422cf1
Normal-4270.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4270.png,Normal/images/Normal-4270.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4270.png,Normal/masks/Normal-4270.png,True,False,299,299,L,132.61,a3bee09d3ea3cae5c751070224d050a4
Normal-4271.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4271.png,Normal/images/Normal-4271.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4271.png,Normal/masks/Normal-4271.png,True,False,299,299,L,104.44,32b9ffc6cae938f678ad12c2009f47a0
Normal-4272.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4272.png,Normal/images/Normal-4272.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4272.png,Normal/masks/Normal-4272.png,True,False,299,299,L,108.2,1a934c0fc0d9d14619697f2827ae8f0e
Normal-4273.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4273.png,Normal/images/Normal-4273.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4273.png,Normal/masks/Normal-4273.png,True,False,299,299,L,115.15,d09d02245f09c28b7e64bbcefa7da208
Normal-4274.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4274.png,Normal/images/Normal-4274.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4274.png,Normal/masks/Normal-4274.png,True,False,299,299,L,104.41,62d1959c25ddb2d4453c6ffcb48ca30e
Normal-4275.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4275.png,Normal/images/Normal-4275.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4275.png,Normal/masks/Normal-4275.png,True,False,299,299,L,118.04,647138d38745ca83dfcef02a6e1be132
Normal-4276.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4276.png,Normal/images/Normal-4276.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4276.png,Normal/masks/Normal-4276.png,True,False,299,299,L,162.85,01f63949074fc17908379d96bd142d8a
Normal-4277.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4277.png,Normal/images/Normal-4277.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4277.png,Normal/masks/Normal-4277.png,True,False,299,299,L,110.85,5eaf021a8c9fc1d945deae35f36f299e
Normal-4278.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4278.png,Normal/images/Normal-4278.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4278.png,Normal/masks/Normal-4278.png,True,False,299,299,L,119.17,6bdb3abf330684d9b8f3b197253999a5
Normal-4279.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4279.png,Normal/images/Normal-4279.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4279.png,Normal/masks/Normal-4279.png,True,False,299,299,L,126.34,7d1b86c88f299a149563f545b67645dc
Normal-428.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-428.png,Normal/images/Normal-428.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-428.png,Normal/masks/Normal-428.png,True,False,299,299,L,109.02,ccfc6794b16c636c522b14d74079040a
Normal-4280.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4280.png,Normal/images/Normal-4280.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4280.png,Normal/masks/Normal-4280.png,True,False,299,299,L,110.58,03407c01a1119bb025df9075ee1c92ca
Normal-4281.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4281.png,Normal/images/Normal-4281.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4281.png,Normal/masks/Normal-4281.png,True,False,299,299,L,115.67,40e082e5338c3356ff8ce7ed4080afbb
Normal-4282.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4282.png,Normal/images/Normal-4282.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4282.png,Normal/masks/Normal-4282.png,True,False,299,299,L,145.42,bc8d3aa9b3dd9b05cc0a965dbfa968c5
Normal-4283.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4283.png,Normal/images/Normal-4283.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4283.png,Normal/masks/Normal-4283.png,True,False,299,299,L,124.72,68c1186b20f49741ba8346f1c058998c
Normal-4284.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4284.png,Normal/images/Normal-4284.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4284.png,Normal/masks/Normal-4284.png,True,False,299,299,L,142.89,d4ad892a56d546a673234270470306b9
Normal-4285.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4285.png,Normal/images/Normal-4285.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4285.png,Normal/masks/Normal-4285.png,True,False,299,299,L,114.43,86fedc25868329781b88eaa5756847cb
Normal-4286.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4286.png,Normal/images/Normal-4286.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4286.png,Normal/masks/Normal-4286.png,True,False,299,299,L,120.05,36260a4af85bd2f6a62c28c146b955f1
Normal-4287.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4287.png,Normal/images/Normal-4287.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4287.png,Normal/masks/Normal-4287.png,True,False,299,299,L,98.84,a48bca3c248db615e084c05f32a60821
Normal-4288.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4288.png,Normal/images/Normal-4288.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4288.png,Normal/masks/Normal-4288.png,True,False,299,299,L,158.46,1d23620bf8a3211b7b42f472c6938185
Normal-4289.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4289.png,Normal/images/Normal-4289.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4289.png,Normal/masks/Normal-4289.png,True,False,299,299,L,144.76,3e818138eb04a567510384fc25b0ceb2
Normal-429.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-429.png,Normal/images/Normal-429.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-429.png,Normal/masks/Normal-429.png,True,False,299,299,L,130.71,b001dd0f8de94eb6e267b91bad08de77
Normal-4290.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4290.png,Normal/images/Normal-4290.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4290.png,Normal/masks/Normal-4290.png,True,False,299,299,L,126.68,17ec5796eb08b65bbf2d1742166d5f98
Normal-4291.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4291.png,Normal/images/Normal-4291.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4291.png,Normal/masks/Normal-4291.png,True,False,299,299,L,145.14,b5b8b2de64db9df27724094ba40cead0
Normal-4292.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4292.png,Normal/images/Normal-4292.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4292.png,Normal/masks/Normal-4292.png,True,False,299,299,L,108.99,ad2312eee0dc7fe20789ee8f4487d54a
Normal-4293.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4293.png,Normal/images/Normal-4293.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4293.png,Normal/masks/Normal-4293.png,True,False,299,299,L,110.48,ac6f6b524389e9f1f4b065eae28d2b7b
Normal-4294.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4294.png,Normal/images/Normal-4294.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4294.png,Normal/masks/Normal-4294.png,True,False,299,299,L,134.58,6bb9d7ce2e310282fa9154fe46d44f38
Normal-4295.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4295.png,Normal/images/Normal-4295.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4295.png,Normal/masks/Normal-4295.png,True,False,299,299,L,133.61,94c7c280800b62eb5a34a080810a3769
Normal-4296.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4296.png,Normal/images/Normal-4296.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4296.png,Normal/masks/Normal-4296.png,True,False,299,299,L,121.73,bfdd9cd7417ce53f48567446a407cd7a
Normal-4297.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4297.png,Normal/images/Normal-4297.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4297.png,Normal/masks/Normal-4297.png,True,False,299,299,L,127.71,b3424703e57892a83ddf81d58b9e80aa
Normal-4298.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4298.png,Normal/images/Normal-4298.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4298.png,Normal/masks/Normal-4298.png,True,False,299,299,L,112.27,6d7dea7b7fe3a2424055207e5a627298
Normal-4299.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4299.png,Normal/images/Normal-4299.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4299.png,Normal/masks/Normal-4299.png,True,False,299,299,L,183.25,a4a837af892fa306713c24201eef52f9
Normal-43.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-43.png,Normal/images/Normal-43.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-43.png,Normal/masks/Normal-43.png,True,False,299,299,L,122.9,655f5874e6d16c7e7715db1f0d29f9b4
Normal-430.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-430.png,Normal/images/Normal-430.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-430.png,Normal/masks/Normal-430.png,True,False,299,299,L,112.43,8fd7f80199d875756ec34727254fecb4
Normal-4300.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4300.png,Normal/images/Normal-4300.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4300.png,Normal/masks/Normal-4300.png,True,False,299,299,L,149.01,afa4b68b380c1b62942ce170f9e12e9d
Normal-4301.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4301.png,Normal/images/Normal-4301.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4301.png,Normal/masks/Normal-4301.png,True,False,299,299,L,112.01,68530a0f068a8bce0c4f80bd73d2c345
Normal-4302.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4302.png,Normal/images/Normal-4302.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4302.png,Normal/masks/Normal-4302.png,True,False,299,299,L,129.41,2406b541c0f84d2e3c3b3cbef6072337
Normal-4303.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4303.png,Normal/images/Normal-4303.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4303.png,Normal/masks/Normal-4303.png,True,False,299,299,L,140.74,3cabede026acdf2d082679d52de4a148
Normal-4304.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4304.png,Normal/images/Normal-4304.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4304.png,Normal/masks/Normal-4304.png,True,False,299,299,L,104.57,f7823e64a6fa98bdc857bf3c0379f8c2
Normal-4305.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4305.png,Normal/images/Normal-4305.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4305.png,Normal/masks/Normal-4305.png,True,False,299,299,L,146.74,cf38a1d4bebd9f8810748973f72c3cc0
Normal-4306.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4306.png,Normal/images/Normal-4306.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4306.png,Normal/masks/Normal-4306.png,True,False,299,299,L,86.35,debd33c02640ff95435540e1f9015168
Normal-4307.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4307.png,Normal/images/Normal-4307.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4307.png,Normal/masks/Normal-4307.png,True,False,299,299,L,121.13,b517321c836c1902364829936e530c9e
Normal-4308.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4308.png,Normal/images/Normal-4308.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4308.png,Normal/masks/Normal-4308.png,True,False,299,299,L,201.85,7bac88cb44cf694aaf65b7aaafac0eb6
Normal-4309.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4309.png,Normal/images/Normal-4309.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4309.png,Normal/masks/Normal-4309.png,True,False,299,299,L,104.25,582c8e3b01c9f9db71fbf3ecc66ad6ae
Normal-431.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-431.png,Normal/images/Normal-431.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-431.png,Normal/masks/Normal-431.png,True,False,299,299,L,123.34,fe41bdc81d1694dc1d7cedb5f70b620e
Normal-4310.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4310.png,Normal/images/Normal-4310.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4310.png,Normal/masks/Normal-4310.png,True,False,299,299,L,102.85,7af483240a52712c23025db4dfd670af
Normal-4311.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4311.png,Normal/images/Normal-4311.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4311.png,Normal/masks/Normal-4311.png,True,False,299,299,L,151.69,f064eff85efb9829e3333b73e464c1ed
Normal-4312.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4312.png,Normal/images/Normal-4312.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4312.png,Normal/masks/Normal-4312.png,True,False,299,299,L,172.27,61f276f7b11324e467e7337033d5b8d1
Normal-4313.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4313.png,Normal/images/Normal-4313.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4313.png,Normal/masks/Normal-4313.png,True,False,299,299,L,164.28,ad6aa1f5210c868ef06f3081ac60d038
Normal-4314.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4314.png,Normal/images/Normal-4314.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4314.png,Normal/masks/Normal-4314.png,True,False,299,299,L,121.81,cf6e0acfe0a00e47a49a82d6128f3334
Normal-4315.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4315.png,Normal/images/Normal-4315.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4315.png,Normal/masks/Normal-4315.png,True,False,299,299,L,94.0,98947cac57071cd26ce13b725189bd95
Normal-4316.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4316.png,Normal/images/Normal-4316.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4316.png,Normal/masks/Normal-4316.png,True,False,299,299,L,97.6,87a679a63a527dbbe3dc0cfa3c5d1592
Normal-4317.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4317.png,Normal/images/Normal-4317.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4317.png,Normal/masks/Normal-4317.png,True,False,299,299,L,128.82,db3127c5b03d1653765e15d03c27c848
Normal-4318.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4318.png,Normal/images/Normal-4318.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4318.png,Normal/masks/Normal-4318.png,True,False,299,299,L,171.05,eb1cfe3c1c74b5ad5108c55c00aad373
Normal-4319.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4319.png,Normal/images/Normal-4319.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4319.png,Normal/masks/Normal-4319.png,True,False,299,299,L,137.91,1f4adaaa567c30f0d5fa58c2e477f30c
Normal-432.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-432.png,Normal/images/Normal-432.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-432.png,Normal/masks/Normal-432.png,True,False,299,299,L,120.01,5f913fbe7d299d568e6fea8750b97bd8
Normal-4320.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4320.png,Normal/images/Normal-4320.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4320.png,Normal/masks/Normal-4320.png,True,False,299,299,L,124.57,a223cc1979f0a5936ce84001139d00ec
Normal-4321.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4321.png,Normal/images/Normal-4321.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4321.png,Normal/masks/Normal-4321.png,True,False,299,299,L,141.41,7adf1342c71d8986e550609af2cd0a0e
Normal-4322.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4322.png,Normal/images/Normal-4322.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4322.png,Normal/masks/Normal-4322.png,True,False,299,299,L,109.06,4ae563f0ad9a9b9969a3ec2739c7e163
Normal-4323.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4323.png,Normal/images/Normal-4323.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4323.png,Normal/masks/Normal-4323.png,True,False,299,299,L,111.27,6fa9800b94046cbe6c229d3c4416caeb
Normal-4324.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4324.png,Normal/images/Normal-4324.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4324.png,Normal/masks/Normal-4324.png,True,False,299,299,L,130.93,577be21603378fd14af338a91f21043a
Normal-4325.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4325.png,Normal/images/Normal-4325.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4325.png,Normal/masks/Normal-4325.png,True,False,299,299,L,165.16,c89c1924d3b869805af7363c9bb8cde3
Normal-4326.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4326.png,Normal/images/Normal-4326.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4326.png,Normal/masks/Normal-4326.png,True,False,299,299,L,129.18,aaa11c5a9679f3aec1e6835baeb9573b
Normal-4327.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4327.png,Normal/images/Normal-4327.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4327.png,Normal/masks/Normal-4327.png,True,False,299,299,L,110.82,67a0e9090814c9d06ff65d5f6c12fa8d
Normal-4328.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4328.png,Normal/images/Normal-4328.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4328.png,Normal/masks/Normal-4328.png,True,False,299,299,L,95.97,046fabefc61379169c9467f350021de7
Normal-4329.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4329.png,Normal/images/Normal-4329.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4329.png,Normal/masks/Normal-4329.png,True,False,299,299,L,116.9,3e87fdfc61cf8576136e511b02d1ef8b
Normal-433.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-433.png,Normal/images/Normal-433.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-433.png,Normal/masks/Normal-433.png,True,False,299,299,L,106.69,a7745bbff0d8820791851c0f52d3f413
Normal-4330.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4330.png,Normal/images/Normal-4330.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4330.png,Normal/masks/Normal-4330.png,True,False,299,299,L,181.53,5dfa58280a2f742b94bdd6e3d80c2764
Normal-4331.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4331.png,Normal/images/Normal-4331.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4331.png,Normal/masks/Normal-4331.png,True,False,299,299,L,121.55,9359b7942252fab1f1fc640a2e1d22a8
Normal-4332.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4332.png,Normal/images/Normal-4332.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4332.png,Normal/masks/Normal-4332.png,True,False,299,299,L,103.31,bb428d1f2c493730891733bab9230b32
Normal-4333.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4333.png,Normal/images/Normal-4333.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4333.png,Normal/masks/Normal-4333.png,True,False,299,299,L,90.6,a5c8fe5eeef37a2c5eac8719fe9e4db7
Normal-4334.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4334.png,Normal/images/Normal-4334.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4334.png,Normal/masks/Normal-4334.png,True,False,299,299,L,137.61,ab28646796a24787129885bd0ac3f077
Normal-4335.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4335.png,Normal/images/Normal-4335.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4335.png,Normal/masks/Normal-4335.png,True,False,299,299,L,116.84,36bd5f197f1c1b3ba66bd8992f405f5e
Normal-4336.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4336.png,Normal/images/Normal-4336.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4336.png,Normal/masks/Normal-4336.png,True,False,299,299,L,135.31,148bf22f8c3161b7244f596dda98fe69
Normal-4337.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4337.png,Normal/images/Normal-4337.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4337.png,Normal/masks/Normal-4337.png,True,False,299,299,L,125.82,63ce6c4b0231ed2f7d9ee1c0e10001b4
Normal-4338.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4338.png,Normal/images/Normal-4338.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4338.png,Normal/masks/Normal-4338.png,True,False,299,299,L,118.6,baf1bf791fa666b25b8e147b5f65458a
Normal-4339.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4339.png,Normal/images/Normal-4339.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4339.png,Normal/masks/Normal-4339.png,True,False,299,299,L,133.46,02e1e85092592ca4c091ebf2ff6855e0
Normal-434.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-434.png,Normal/images/Normal-434.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-434.png,Normal/masks/Normal-434.png,True,False,299,299,L,116.7,989b3066f9d49a3faf27277fa587f0fe
Normal-4340.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4340.png,Normal/images/Normal-4340.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4340.png,Normal/masks/Normal-4340.png,True,False,299,299,L,112.5,43833eb92661d720265edc68d6e479da
Normal-4341.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4341.png,Normal/images/Normal-4341.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4341.png,Normal/masks/Normal-4341.png,True,False,299,299,L,102.35,699098ac82ad817bea0c6070635eda7b
Normal-4342.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4342.png,Normal/images/Normal-4342.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4342.png,Normal/masks/Normal-4342.png,True,False,299,299,L,114.42,60c3c6974b85955a3ae61095127d064d
Normal-4343.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4343.png,Normal/images/Normal-4343.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4343.png,Normal/masks/Normal-4343.png,True,False,299,299,L,145.27,04a145fa7d394aa52fd4ee9888c5b335
Normal-4344.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4344.png,Normal/images/Normal-4344.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4344.png,Normal/masks/Normal-4344.png,True,False,299,299,L,116.95,c4a3646bf35abe01fc5de5d8d11f4e85
Normal-4345.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4345.png,Normal/images/Normal-4345.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4345.png,Normal/masks/Normal-4345.png,True,False,299,299,L,110.56,76867cc38a6bf7fef6e6814b8fb12c83
Normal-4346.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4346.png,Normal/images/Normal-4346.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4346.png,Normal/masks/Normal-4346.png,True,False,299,299,L,134.47,a2a1c336191a6ce2878376a1e547c6ab
Normal-4347.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4347.png,Normal/images/Normal-4347.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4347.png,Normal/masks/Normal-4347.png,True,False,299,299,L,123.98,5f0b2d1b4a6e053282be8ef158beb3b9
Normal-4348.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4348.png,Normal/images/Normal-4348.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4348.png,Normal/masks/Normal-4348.png,True,False,299,299,L,118.3,da278369b9d1380fb15f3b92ac54d6dc
Normal-4349.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4349.png,Normal/images/Normal-4349.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4349.png,Normal/masks/Normal-4349.png,True,False,299,299,L,93.25,aab843bc1ad645f5fa21e44a7665a311
Normal-435.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-435.png,Normal/images/Normal-435.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-435.png,Normal/masks/Normal-435.png,True,False,299,299,L,121.1,d3dfd71e3dd8ae8ad29c9bb0afab4d3d
Normal-4350.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4350.png,Normal/images/Normal-4350.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4350.png,Normal/masks/Normal-4350.png,True,False,299,299,L,127.53,5436f0569bb98561b376b0893ab49954
Normal-4351.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4351.png,Normal/images/Normal-4351.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4351.png,Normal/masks/Normal-4351.png,True,False,299,299,L,120.23,b736f06b082efd21df1d52b36a8ec6d1
Normal-4352.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4352.png,Normal/images/Normal-4352.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4352.png,Normal/masks/Normal-4352.png,True,False,299,299,L,112.37,b2768ff38a36a248598d9c9c43916d9d
Normal-4353.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4353.png,Normal/images/Normal-4353.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4353.png,Normal/masks/Normal-4353.png,True,False,299,299,L,114.11,99366c1aac0229e73e7056dfc469dfa8
Normal-4354.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4354.png,Normal/images/Normal-4354.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4354.png,Normal/masks/Normal-4354.png,True,False,299,299,L,143.77,86baa1578983d56f9042fcda2d3b9e5f
Normal-4355.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4355.png,Normal/images/Normal-4355.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4355.png,Normal/masks/Normal-4355.png,True,False,299,299,L,144.77,d6b612a33b2122bdf99b650d277d3688
Normal-4356.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4356.png,Normal/images/Normal-4356.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4356.png,Normal/masks/Normal-4356.png,True,False,299,299,L,169.01,3c41f48ec8adcdf5a347d0eef11c1c3d
Normal-4357.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4357.png,Normal/images/Normal-4357.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4357.png,Normal/masks/Normal-4357.png,True,False,299,299,L,164.2,0c523b5e6af64d83a70f78e90619dca7
Normal-4358.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4358.png,Normal/images/Normal-4358.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4358.png,Normal/masks/Normal-4358.png,True,False,299,299,L,120.06,e234faa4dcea19e0dc2dc0df4d29db24
Normal-4359.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4359.png,Normal/images/Normal-4359.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4359.png,Normal/masks/Normal-4359.png,True,False,299,299,L,132.88,839b2e07b3ea11989606b271a9e3213b
Normal-436.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-436.png,Normal/images/Normal-436.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-436.png,Normal/masks/Normal-436.png,True,False,299,299,L,108.59,9646353a842d3d4734031b60e9cabbc7
Normal-4360.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4360.png,Normal/images/Normal-4360.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4360.png,Normal/masks/Normal-4360.png,True,False,299,299,L,118.54,64cba68704aa17fec89e0950c6ac53b6
Normal-4361.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4361.png,Normal/images/Normal-4361.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4361.png,Normal/masks/Normal-4361.png,True,False,299,299,L,144.34,329d82af6c15b6ae3810fba5def4a87f
Normal-4362.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4362.png,Normal/images/Normal-4362.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4362.png,Normal/masks/Normal-4362.png,True,False,299,299,L,149.5,19aa4facd2e26b9452e9820934731171
Normal-4363.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4363.png,Normal/images/Normal-4363.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4363.png,Normal/masks/Normal-4363.png,True,False,299,299,L,108.31,e28d484143e0316dc32753d4d1b505df
Normal-4364.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4364.png,Normal/images/Normal-4364.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4364.png,Normal/masks/Normal-4364.png,True,False,299,299,L,112.44,437dd720d6d90bcfc8c3c716ae3a8742
Normal-4365.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4365.png,Normal/images/Normal-4365.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4365.png,Normal/masks/Normal-4365.png,True,False,299,299,L,125.66,113db1e5bb6d428c84f7b7de5efd57ba
Normal-4366.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4366.png,Normal/images/Normal-4366.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4366.png,Normal/masks/Normal-4366.png,True,False,299,299,L,119.71,447de78e7eda46798b777c253262cf60
Normal-4367.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4367.png,Normal/images/Normal-4367.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4367.png,Normal/masks/Normal-4367.png,True,False,299,299,L,166.43,b437fc1879f14ea3cc43626e03c8217b
Normal-4368.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4368.png,Normal/images/Normal-4368.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4368.png,Normal/masks/Normal-4368.png,True,False,299,299,L,121.77,01b292469b56fcd1d886743bfdf0a7d8
Normal-4369.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4369.png,Normal/images/Normal-4369.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4369.png,Normal/masks/Normal-4369.png,True,False,299,299,L,147.93,df64fae266fda76dc5d50166d57c9d42
Normal-437.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-437.png,Normal/images/Normal-437.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-437.png,Normal/masks/Normal-437.png,True,False,299,299,L,159.24,85fd6f31818609e654889b95969d6fbb
Normal-4370.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4370.png,Normal/images/Normal-4370.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4370.png,Normal/masks/Normal-4370.png,True,False,299,299,L,128.2,2bfbbaef0961d044c245bb6326967051
Normal-4371.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4371.png,Normal/images/Normal-4371.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4371.png,Normal/masks/Normal-4371.png,True,False,299,299,L,115.32,cfbdace41c87215c56c54fc41e7407b7
Normal-4372.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4372.png,Normal/images/Normal-4372.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4372.png,Normal/masks/Normal-4372.png,True,False,299,299,L,124.09,52082bcba74f478416141ea9498eeedc
Normal-4373.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4373.png,Normal/images/Normal-4373.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4373.png,Normal/masks/Normal-4373.png,True,False,299,299,L,173.25,a4d3c5be73d775f508a7b7a5034458a6
Normal-4374.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4374.png,Normal/images/Normal-4374.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4374.png,Normal/masks/Normal-4374.png,True,False,299,299,L,139.46,3c3b3ec2e319bc04bb0c6eb961c14255
Normal-4375.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4375.png,Normal/images/Normal-4375.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4375.png,Normal/masks/Normal-4375.png,True,False,299,299,L,118.83,a7919654fd9d5e24681de4864dda1937
Normal-4376.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4376.png,Normal/images/Normal-4376.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4376.png,Normal/masks/Normal-4376.png,True,False,299,299,L,131.72,d784ff9826c55fa906c173474c27cb73
Normal-4377.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4377.png,Normal/images/Normal-4377.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4377.png,Normal/masks/Normal-4377.png,True,False,299,299,L,125.47,b1640f71b525b96be86d7178a111f1f6
Normal-4378.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4378.png,Normal/images/Normal-4378.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4378.png,Normal/masks/Normal-4378.png,True,False,299,299,L,161.4,14dd3084a37694607e306eef4861af70
Normal-4379.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4379.png,Normal/images/Normal-4379.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4379.png,Normal/masks/Normal-4379.png,True,False,299,299,L,130.01,b185d595fc9bc556edb667cfc23a83a2
Normal-438.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-438.png,Normal/images/Normal-438.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-438.png,Normal/masks/Normal-438.png,True,False,299,299,L,117.91,9324b7798840435028aa92e11e7710eb
Normal-4380.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4380.png,Normal/images/Normal-4380.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4380.png,Normal/masks/Normal-4380.png,True,False,299,299,L,100.13,5be4c907f07c45aa11239522a5f17417
Normal-4381.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4381.png,Normal/images/Normal-4381.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4381.png,Normal/masks/Normal-4381.png,True,False,299,299,L,111.45,ea8e5885d998395a1c65fe381921357e
Normal-4382.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4382.png,Normal/images/Normal-4382.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4382.png,Normal/masks/Normal-4382.png,True,False,299,299,L,160.04,fbaf34d71151a92090ebcd3840799e58
Normal-4383.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4383.png,Normal/images/Normal-4383.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4383.png,Normal/masks/Normal-4383.png,True,False,299,299,L,112.31,54d8038d587cc1456150a01ffc0a8935
Normal-4384.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4384.png,Normal/images/Normal-4384.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4384.png,Normal/masks/Normal-4384.png,True,False,299,299,L,126.01,b95b4d9b00ecdb46c1cfcca9e91d55d7
Normal-4385.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4385.png,Normal/images/Normal-4385.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4385.png,Normal/masks/Normal-4385.png,True,False,299,299,L,148.85,545599e91d000a1c5d3d17ac6dd2beeb
Normal-4386.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4386.png,Normal/images/Normal-4386.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4386.png,Normal/masks/Normal-4386.png,True,False,299,299,L,122.45,6955d6587e68ceb6129ff957f0ff78bb
Normal-4387.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4387.png,Normal/images/Normal-4387.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4387.png,Normal/masks/Normal-4387.png,True,False,299,299,L,146.77,731255d48613293a9eb19e9898c60623
Normal-4388.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4388.png,Normal/images/Normal-4388.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4388.png,Normal/masks/Normal-4388.png,True,False,299,299,L,145.0,76369ded922b708666b26d84d4f19bf3
Normal-4389.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4389.png,Normal/images/Normal-4389.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4389.png,Normal/masks/Normal-4389.png,True,False,299,299,L,126.84,b16607b26683ce125574c22bbdb44262
Normal-439.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-439.png,Normal/images/Normal-439.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-439.png,Normal/masks/Normal-439.png,True,False,299,299,L,115.43,0f221bb9698e771c9dd87aad58ba37d6
Normal-4390.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4390.png,Normal/images/Normal-4390.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4390.png,Normal/masks/Normal-4390.png,True,False,299,299,L,170.48,005b7ce6d6fe392db7aaec957717e063
Normal-4391.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4391.png,Normal/images/Normal-4391.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4391.png,Normal/masks/Normal-4391.png,True,False,299,299,L,125.42,efcff34a9d53943022b15eb44530b4dc
Normal-4392.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4392.png,Normal/images/Normal-4392.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4392.png,Normal/masks/Normal-4392.png,True,False,299,299,L,82.1,a29cac28e3958a7dd88bec74e64a1ce8
Normal-4393.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4393.png,Normal/images/Normal-4393.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4393.png,Normal/masks/Normal-4393.png,True,False,299,299,L,170.93,1093a15211291527ad21dbcbf663629c
Normal-4394.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4394.png,Normal/images/Normal-4394.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4394.png,Normal/masks/Normal-4394.png,True,False,299,299,L,151.82,1f6942684fe4cdb5db5a4c1471028be8
Normal-4395.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4395.png,Normal/images/Normal-4395.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4395.png,Normal/masks/Normal-4395.png,True,False,299,299,L,95.06,9aadfbd612ca4778abb04afcaad5a019
Normal-4396.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4396.png,Normal/images/Normal-4396.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4396.png,Normal/masks/Normal-4396.png,True,False,299,299,L,105.91,73874cba985e68d9be5b46d5cefac309
Normal-4397.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4397.png,Normal/images/Normal-4397.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4397.png,Normal/masks/Normal-4397.png,True,False,299,299,L,141.61,d6fa81ab2025479559b12b31a5a62d6b
Normal-4398.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4398.png,Normal/images/Normal-4398.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4398.png,Normal/masks/Normal-4398.png,True,False,299,299,L,113.72,cb61af49e7c0f4da8fd136f7be8f5809
Normal-4399.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4399.png,Normal/images/Normal-4399.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4399.png,Normal/masks/Normal-4399.png,True,False,299,299,L,133.66,cdcdd59c5fb96496573999d67ec010ee
Normal-44.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-44.png,Normal/images/Normal-44.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-44.png,Normal/masks/Normal-44.png,True,False,299,299,L,150.8,75400d0dfa7d5556fd3913bd7de1952c
Normal-440.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-440.png,Normal/images/Normal-440.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-440.png,Normal/masks/Normal-440.png,True,False,299,299,L,125.79,0ff972babd0f9efc3d01f8464227f6b7
Normal-4400.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4400.png,Normal/images/Normal-4400.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4400.png,Normal/masks/Normal-4400.png,True,False,299,299,L,119.28,5d0ed8b98dacb60b02eeb39fc5982359
Normal-4401.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4401.png,Normal/images/Normal-4401.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4401.png,Normal/masks/Normal-4401.png,True,False,299,299,L,128.71,7ab5cdb3e3e16f934892dd21a2f5a13c
Normal-4402.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4402.png,Normal/images/Normal-4402.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4402.png,Normal/masks/Normal-4402.png,True,False,299,299,L,111.26,63c29f573ef6969a501448cf2745b78a
Normal-4403.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4403.png,Normal/images/Normal-4403.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4403.png,Normal/masks/Normal-4403.png,True,False,299,299,L,131.35,49fcb17e8020b6355cec84fa44a640dd
Normal-4404.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4404.png,Normal/images/Normal-4404.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4404.png,Normal/masks/Normal-4404.png,True,False,299,299,L,111.04,2f20db8a54bce442634ca05f581990e6
Normal-4405.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4405.png,Normal/images/Normal-4405.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4405.png,Normal/masks/Normal-4405.png,True,False,299,299,L,113.83,0555d06d0facbf818b41df561fbebdbb
Normal-4406.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4406.png,Normal/images/Normal-4406.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4406.png,Normal/masks/Normal-4406.png,True,False,299,299,L,127.57,1cd3e8b2619fde7594f8b101e599056f
Normal-4407.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4407.png,Normal/images/Normal-4407.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4407.png,Normal/masks/Normal-4407.png,True,False,299,299,L,122.37,d0185cf7ca9347a6c243f1de2c3805b6
Normal-4408.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4408.png,Normal/images/Normal-4408.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4408.png,Normal/masks/Normal-4408.png,True,False,299,299,L,159.95,38fe7441b85712aab1450b3ae0c7ade9
Normal-4409.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4409.png,Normal/images/Normal-4409.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4409.png,Normal/masks/Normal-4409.png,True,False,299,299,L,139.52,58e6ba9e5d6c5ebf83d43de916203a3d
Normal-441.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-441.png,Normal/images/Normal-441.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-441.png,Normal/masks/Normal-441.png,True,False,299,299,L,126.4,103b8f103eafd5bec2ab216232989887
Normal-4410.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4410.png,Normal/images/Normal-4410.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4410.png,Normal/masks/Normal-4410.png,True,False,299,299,L,111.17,020dd1b932a44ee84334e87a1d0ae121
Normal-4411.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4411.png,Normal/images/Normal-4411.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4411.png,Normal/masks/Normal-4411.png,True,False,299,299,L,115.85,58936edd29b2847ef08e359cd09da796
Normal-4412.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4412.png,Normal/images/Normal-4412.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4412.png,Normal/masks/Normal-4412.png,True,False,299,299,L,122.74,c711b7f1b251c91cfe94bdfb23e047e6
Normal-4413.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4413.png,Normal/images/Normal-4413.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4413.png,Normal/masks/Normal-4413.png,True,False,299,299,L,135.61,a60f40772e84a00f03b9ae0b120c6b13
Normal-4414.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4414.png,Normal/images/Normal-4414.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4414.png,Normal/masks/Normal-4414.png,True,False,299,299,L,99.81,abc2114bb773c3ccc6e9dae7107dfa1e
Normal-4415.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4415.png,Normal/images/Normal-4415.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4415.png,Normal/masks/Normal-4415.png,True,False,299,299,L,138.19,f9979a80a00549dd65174eb3ef8ab3da
Normal-4416.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4416.png,Normal/images/Normal-4416.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4416.png,Normal/masks/Normal-4416.png,True,False,299,299,L,133.96,012f590acb81c1b992030962e75ff6c5
Normal-4417.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4417.png,Normal/images/Normal-4417.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4417.png,Normal/masks/Normal-4417.png,True,False,299,299,L,132.43,cd280845ff6152e920c7f3b2f8e040dd
Normal-4418.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4418.png,Normal/images/Normal-4418.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4418.png,Normal/masks/Normal-4418.png,True,False,299,299,L,150.66,46666b54581b8a5a60dfc8629a861b95
Normal-4419.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4419.png,Normal/images/Normal-4419.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4419.png,Normal/masks/Normal-4419.png,True,False,299,299,L,136.08,33c5b3fb2205c79855c2d353c42e4242
Normal-442.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-442.png,Normal/images/Normal-442.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-442.png,Normal/masks/Normal-442.png,True,False,299,299,L,146.31,7e89637b5115d96e439f5f7a0be9b619
Normal-4420.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4420.png,Normal/images/Normal-4420.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4420.png,Normal/masks/Normal-4420.png,True,False,299,299,L,131.69,35acc139241340b3cc709b134e79a528
Normal-4421.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4421.png,Normal/images/Normal-4421.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4421.png,Normal/masks/Normal-4421.png,True,False,299,299,L,151.88,c40e6655faf8c4ffe89848d04c90aeac
Normal-4422.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4422.png,Normal/images/Normal-4422.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4422.png,Normal/masks/Normal-4422.png,True,False,299,299,L,111.93,d573a529ff60c0f45078dda8ddce1e27
Normal-4423.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4423.png,Normal/images/Normal-4423.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4423.png,Normal/masks/Normal-4423.png,True,False,299,299,L,108.75,ab9526ebc19c4673adcc5e01d4356fdc
Normal-4424.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4424.png,Normal/images/Normal-4424.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4424.png,Normal/masks/Normal-4424.png,True,False,299,299,L,113.89,2b959717f5851b3ffd7b525d17a547cf
Normal-4425.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4425.png,Normal/images/Normal-4425.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4425.png,Normal/masks/Normal-4425.png,True,False,299,299,L,119.15,8ebd4892d464d0bb2502dde36d42916f
Normal-4426.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4426.png,Normal/images/Normal-4426.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4426.png,Normal/masks/Normal-4426.png,True,False,299,299,L,135.15,e22042e50c0d61510bc97dff849ed5c1
Normal-4427.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4427.png,Normal/images/Normal-4427.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4427.png,Normal/masks/Normal-4427.png,True,False,299,299,L,75.23,96a524f1b002f8253657c5c4b187b499
Normal-4428.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4428.png,Normal/images/Normal-4428.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4428.png,Normal/masks/Normal-4428.png,True,False,299,299,L,114.76,01a37f8f10407866b978cd33e66a871d
Normal-4429.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4429.png,Normal/images/Normal-4429.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4429.png,Normal/masks/Normal-4429.png,True,False,299,299,L,113.63,3efddceae71c59cdf02f5346de2878e9
Normal-443.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-443.png,Normal/images/Normal-443.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-443.png,Normal/masks/Normal-443.png,True,False,299,299,L,116.62,0cb00212e39db696fe895614d4a21b56
Normal-4430.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4430.png,Normal/images/Normal-4430.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4430.png,Normal/masks/Normal-4430.png,True,False,299,299,L,163.6,1b2dd17a133e63c0aa4d7598d4d7ed5a
Normal-4431.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4431.png,Normal/images/Normal-4431.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4431.png,Normal/masks/Normal-4431.png,True,False,299,299,L,109.88,fc65eb60330664265a2a35002a4036e1
Normal-4432.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4432.png,Normal/images/Normal-4432.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4432.png,Normal/masks/Normal-4432.png,True,False,299,299,L,170.77,236ce61b30b12dc2a281b03b6d8cbf42
Normal-4433.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4433.png,Normal/images/Normal-4433.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4433.png,Normal/masks/Normal-4433.png,True,False,299,299,L,166.23,0c810884db7d0342b1ebe95ea1db08b7
Normal-4434.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4434.png,Normal/images/Normal-4434.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4434.png,Normal/masks/Normal-4434.png,True,False,299,299,L,114.17,c0449fd2727397b8887e34999ae069b5
Normal-4435.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4435.png,Normal/images/Normal-4435.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4435.png,Normal/masks/Normal-4435.png,True,False,299,299,L,119.91,3617f4ada04c741376dcf460f593f73d
Normal-4436.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4436.png,Normal/images/Normal-4436.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4436.png,Normal/masks/Normal-4436.png,True,False,299,299,L,122.4,7f8f4c9d4d1b1622f834f538869d1d01
Normal-4437.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4437.png,Normal/images/Normal-4437.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4437.png,Normal/masks/Normal-4437.png,True,False,299,299,L,156.65,997bbe173559f59718f01bbb330a282e
Normal-4438.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4438.png,Normal/images/Normal-4438.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4438.png,Normal/masks/Normal-4438.png,True,False,299,299,L,80.03,c789c9b9adb22028654e9321a8a10889
Normal-4439.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4439.png,Normal/images/Normal-4439.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4439.png,Normal/masks/Normal-4439.png,True,False,299,299,L,116.64,eb6342309b0129b70f41512ba0b30efe
Normal-444.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-444.png,Normal/images/Normal-444.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-444.png,Normal/masks/Normal-444.png,True,False,299,299,L,122.58,abe61915bd7cdebdf445467db5673cc1
Normal-4440.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4440.png,Normal/images/Normal-4440.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4440.png,Normal/masks/Normal-4440.png,True,False,299,299,L,129.91,17b3e64804fab0000ec6ce19878ff6cd
Normal-4441.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4441.png,Normal/images/Normal-4441.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4441.png,Normal/masks/Normal-4441.png,True,False,299,299,L,146.57,f870ee8c6e283a15df45704c0c2af744
Normal-4442.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4442.png,Normal/images/Normal-4442.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4442.png,Normal/masks/Normal-4442.png,True,False,299,299,L,156.68,d6c8e9cd33f4eed75c4e0bf2bcf69b0f
Normal-4443.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4443.png,Normal/images/Normal-4443.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4443.png,Normal/masks/Normal-4443.png,True,False,299,299,L,161.31,39580b580e5e42f3cf5f43e45a8e86eb
Normal-4444.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4444.png,Normal/images/Normal-4444.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4444.png,Normal/masks/Normal-4444.png,True,False,299,299,L,145.36,90949b479e407d6f2b1823691784d2a8
Normal-4445.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4445.png,Normal/images/Normal-4445.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4445.png,Normal/masks/Normal-4445.png,True,False,299,299,L,159.18,ed45ed11340adb22cf23e6e9184f3096
Normal-4446.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4446.png,Normal/images/Normal-4446.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4446.png,Normal/masks/Normal-4446.png,True,False,299,299,L,106.18,7ce3b7f20b8dab7d264fa901293544f4
Normal-4447.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4447.png,Normal/images/Normal-4447.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4447.png,Normal/masks/Normal-4447.png,True,False,299,299,L,119.46,55d35154f030e29362098c0abd1ae901
Normal-4448.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4448.png,Normal/images/Normal-4448.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4448.png,Normal/masks/Normal-4448.png,True,False,299,299,L,116.56,aa3032bd980994723551ebda7c5af262
Normal-4449.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4449.png,Normal/images/Normal-4449.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4449.png,Normal/masks/Normal-4449.png,True,False,299,299,L,99.14,7a7ef96665313703fe70625ff00eb4ed
Normal-445.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-445.png,Normal/images/Normal-445.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-445.png,Normal/masks/Normal-445.png,True,False,299,299,L,122.02,3e86ee7e447504bafb3843d0ab87bbb6
Normal-4450.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4450.png,Normal/images/Normal-4450.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4450.png,Normal/masks/Normal-4450.png,True,False,299,299,L,125.8,8cbef7e853fc1ae376664ca56e6dc329
Normal-4451.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4451.png,Normal/images/Normal-4451.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4451.png,Normal/masks/Normal-4451.png,True,False,299,299,L,170.01,ba500e41be5c585bf123b88a4701230c
Normal-4452.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4452.png,Normal/images/Normal-4452.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4452.png,Normal/masks/Normal-4452.png,True,False,299,299,L,102.04,d40b1883d4fdaa67f09c8aa044a39859
Normal-4453.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4453.png,Normal/images/Normal-4453.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4453.png,Normal/masks/Normal-4453.png,True,False,299,299,L,156.06,ce1f4a1b2e1bef7e516d109b6ebb48a3
Normal-4454.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4454.png,Normal/images/Normal-4454.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4454.png,Normal/masks/Normal-4454.png,True,False,299,299,L,169.55,55a07c7704cf8d72b41a0934b975d396
Normal-4455.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4455.png,Normal/images/Normal-4455.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4455.png,Normal/masks/Normal-4455.png,True,False,299,299,L,138.61,00d459221dbe877376e9c919e08d8af1
Normal-4456.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4456.png,Normal/images/Normal-4456.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4456.png,Normal/masks/Normal-4456.png,True,False,299,299,L,120.02,bcb7897df2becc14867766d0873fa003
Normal-4457.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4457.png,Normal/images/Normal-4457.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4457.png,Normal/masks/Normal-4457.png,True,False,299,299,L,115.85,7cb972a9ef409aa400ab02a65dea5c0c
Normal-4458.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4458.png,Normal/images/Normal-4458.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4458.png,Normal/masks/Normal-4458.png,True,False,299,299,L,106.06,937320576f1d96e8f93ab6ac0f234d1f
Normal-4459.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4459.png,Normal/images/Normal-4459.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4459.png,Normal/masks/Normal-4459.png,True,False,299,299,L,114.28,be728fa3af8f471cc5fce873f2b0fc9b
Normal-446.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-446.png,Normal/images/Normal-446.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-446.png,Normal/masks/Normal-446.png,True,False,299,299,L,133.72,f4386db3474658ddb828b7accc5314f5
Normal-4460.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4460.png,Normal/images/Normal-4460.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4460.png,Normal/masks/Normal-4460.png,True,False,299,299,L,154.76,515ac5acddc003aa25d5655a27e01bf8
Normal-4461.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4461.png,Normal/images/Normal-4461.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4461.png,Normal/masks/Normal-4461.png,True,False,299,299,L,136.03,3759c966f91cde8ca06231741ba8d977
Normal-4462.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4462.png,Normal/images/Normal-4462.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4462.png,Normal/masks/Normal-4462.png,True,False,299,299,L,165.43,2a0a2da2b22ea8a479e1c63952b0daa3
Normal-4463.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4463.png,Normal/images/Normal-4463.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4463.png,Normal/masks/Normal-4463.png,True,False,299,299,L,98.41,de044df7d30c11cb05c54356bad9774f
Normal-4464.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4464.png,Normal/images/Normal-4464.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4464.png,Normal/masks/Normal-4464.png,True,False,299,299,L,167.21,ad1b82c32bff9e7ea2a0af205057c657
Normal-4465.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4465.png,Normal/images/Normal-4465.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4465.png,Normal/masks/Normal-4465.png,True,False,299,299,L,107.59,76d60fa21c0c112a40040234b1051d66
Normal-4466.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4466.png,Normal/images/Normal-4466.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4466.png,Normal/masks/Normal-4466.png,True,False,299,299,L,150.97,065b72d58f2d97d20f9a69a302ed82da
Normal-4467.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4467.png,Normal/images/Normal-4467.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4467.png,Normal/masks/Normal-4467.png,True,False,299,299,L,120.7,3ee1c635ed96c838931ab1c756e40d45
Normal-4468.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4468.png,Normal/images/Normal-4468.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4468.png,Normal/masks/Normal-4468.png,True,False,299,299,L,129.15,caf789d630a9a194f35357535f0c1509
Normal-4469.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4469.png,Normal/images/Normal-4469.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4469.png,Normal/masks/Normal-4469.png,True,False,299,299,L,93.68,3ef86372772336f7b2da31858daf94e6
Normal-447.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-447.png,Normal/images/Normal-447.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-447.png,Normal/masks/Normal-447.png,True,False,299,299,L,113.59,c82b8b5cc5150bd25aac9883975b01b4
Normal-4470.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4470.png,Normal/images/Normal-4470.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4470.png,Normal/masks/Normal-4470.png,True,False,299,299,L,158.0,8e557c989b3c020632aeccc536d94fd5
Normal-4471.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4471.png,Normal/images/Normal-4471.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4471.png,Normal/masks/Normal-4471.png,True,False,299,299,L,133.84,7b2a21a7cea39b47b2b87bc4ac422c88
Normal-4472.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4472.png,Normal/images/Normal-4472.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4472.png,Normal/masks/Normal-4472.png,True,False,299,299,L,108.55,3b1b559bf05685e32b43bac99f376d55
Normal-4473.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4473.png,Normal/images/Normal-4473.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4473.png,Normal/masks/Normal-4473.png,True,False,299,299,L,130.01,f0fa57e9acd4c609b6db321e24507bbb
Normal-4474.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4474.png,Normal/images/Normal-4474.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4474.png,Normal/masks/Normal-4474.png,True,False,299,299,L,167.32,6efc1248e0051402e962567346a9809a
Normal-4475.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4475.png,Normal/images/Normal-4475.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4475.png,Normal/masks/Normal-4475.png,True,False,299,299,L,152.69,51a9af1b5077b3cc62178b5fd5f6228f
Normal-4476.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4476.png,Normal/images/Normal-4476.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4476.png,Normal/masks/Normal-4476.png,True,False,299,299,L,120.26,709656afc333e5ce8b0ee3a101258f2a
Normal-4477.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4477.png,Normal/images/Normal-4477.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4477.png,Normal/masks/Normal-4477.png,True,False,299,299,L,142.95,5bc46ce9124934e068100c6d7dffa8b3
Normal-4478.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4478.png,Normal/images/Normal-4478.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4478.png,Normal/masks/Normal-4478.png,True,False,299,299,L,129.65,b32e33401d30092256c94d2a7b9d482c
Normal-4479.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4479.png,Normal/images/Normal-4479.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4479.png,Normal/masks/Normal-4479.png,True,False,299,299,L,128.85,e500f5df832b188203c3ffe0d0e33293
Normal-448.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-448.png,Normal/images/Normal-448.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-448.png,Normal/masks/Normal-448.png,True,False,299,299,L,136.6,384685699721e9fda1a497e3f22786f0
Normal-4480.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4480.png,Normal/images/Normal-4480.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4480.png,Normal/masks/Normal-4480.png,True,False,299,299,L,143.83,7306993a79d019fd4ec567b543aa8fd6
Normal-4481.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4481.png,Normal/images/Normal-4481.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4481.png,Normal/masks/Normal-4481.png,True,False,299,299,L,133.45,330d158fe10cef46eb38269e7ca106fa
Normal-4482.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4482.png,Normal/images/Normal-4482.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4482.png,Normal/masks/Normal-4482.png,True,False,299,299,L,111.07,10e762ea09e140b96a08096abc876ae5
Normal-4483.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4483.png,Normal/images/Normal-4483.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4483.png,Normal/masks/Normal-4483.png,True,False,299,299,L,114.52,8a1c6000d3495db9d9d9944469a69368
Normal-4484.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4484.png,Normal/images/Normal-4484.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4484.png,Normal/masks/Normal-4484.png,True,False,299,299,L,155.51,28c00bac382cc33bba5e78ab396f959d
Normal-4485.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4485.png,Normal/images/Normal-4485.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4485.png,Normal/masks/Normal-4485.png,True,False,299,299,L,131.61,d2bc8b532b93832ff43a5f3259cf7969
Normal-4486.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4486.png,Normal/images/Normal-4486.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4486.png,Normal/masks/Normal-4486.png,True,False,299,299,L,174.21,6fb9132aa76c1d65ad1a1884cf7f1656
Normal-4487.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4487.png,Normal/images/Normal-4487.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4487.png,Normal/masks/Normal-4487.png,True,False,299,299,L,144.43,77e427e30448f11fbd9c538079e07c3b
Normal-4488.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4488.png,Normal/images/Normal-4488.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4488.png,Normal/masks/Normal-4488.png,True,False,299,299,L,140.2,6024c9a3993a5248ff79691f26d04ff8
Normal-4489.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4489.png,Normal/images/Normal-4489.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4489.png,Normal/masks/Normal-4489.png,True,False,299,299,L,117.92,fdb2818a9a2cb25fd1c95978e7f4032f
Normal-449.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-449.png,Normal/images/Normal-449.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-449.png,Normal/masks/Normal-449.png,True,False,299,299,L,120.59,19487a69e501672b74c636e0794554a0
Normal-4490.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4490.png,Normal/images/Normal-4490.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4490.png,Normal/masks/Normal-4490.png,True,False,299,299,L,109.88,5950db20c8c30fad0ca15f1b620fbf59
Normal-4491.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4491.png,Normal/images/Normal-4491.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4491.png,Normal/masks/Normal-4491.png,True,False,299,299,L,110.55,a53c9655bc00c9c8961fcaa0556cfdd4
Normal-4492.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4492.png,Normal/images/Normal-4492.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4492.png,Normal/masks/Normal-4492.png,True,False,299,299,L,112.45,0a57ba345f7a9298b9fb03437adb5008
Normal-4493.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4493.png,Normal/images/Normal-4493.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4493.png,Normal/masks/Normal-4493.png,True,False,299,299,L,120.26,56f19e0bf43b30a7acafbcacaeff529f
Normal-4494.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4494.png,Normal/images/Normal-4494.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4494.png,Normal/masks/Normal-4494.png,True,False,299,299,L,121.98,435de05b859e8157de58e3ede0908f33
Normal-4495.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4495.png,Normal/images/Normal-4495.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4495.png,Normal/masks/Normal-4495.png,True,False,299,299,L,137.64,0703dbdfa5bdceb7afa986dcfcfb380e
Normal-4496.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4496.png,Normal/images/Normal-4496.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4496.png,Normal/masks/Normal-4496.png,True,False,299,299,L,137.2,a685042cc621fa0aab390ee5d5106128
Normal-4497.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4497.png,Normal/images/Normal-4497.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4497.png,Normal/masks/Normal-4497.png,True,False,299,299,L,128.57,74a926d4d25948de2eeafeaffd730d07
Normal-4498.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4498.png,Normal/images/Normal-4498.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4498.png,Normal/masks/Normal-4498.png,True,False,299,299,L,102.82,b0ad03eeb9e9b1c47f87549ecdd32f56
Normal-4499.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4499.png,Normal/images/Normal-4499.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4499.png,Normal/masks/Normal-4499.png,True,False,299,299,L,142.4,6c150265cafcd87a7f4372c28314996a
Normal-45.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-45.png,Normal/images/Normal-45.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-45.png,Normal/masks/Normal-45.png,True,False,299,299,L,175.82,0ac56d1d11bb8383f6a2ab8dd6b6b7d0
Normal-450.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-450.png,Normal/images/Normal-450.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-450.png,Normal/masks/Normal-450.png,True,False,299,299,L,105.03,8974a77270477bef1df051a90de8d507
Normal-4500.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4500.png,Normal/images/Normal-4500.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4500.png,Normal/masks/Normal-4500.png,True,False,299,299,L,112.03,3fb87ab26434a0e6a809f86c93b976a8
Normal-4501.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4501.png,Normal/images/Normal-4501.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4501.png,Normal/masks/Normal-4501.png,True,False,299,299,L,125.8,4f28acccc623191a82d324e5550652e3
Normal-4502.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4502.png,Normal/images/Normal-4502.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4502.png,Normal/masks/Normal-4502.png,True,False,299,299,L,172.43,9526ba372503c276099bb106b4fd9610
Normal-4503.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4503.png,Normal/images/Normal-4503.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4503.png,Normal/masks/Normal-4503.png,True,False,299,299,L,125.42,b7f59eb13743390a48286a33e07b0fc6
Normal-4504.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4504.png,Normal/images/Normal-4504.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4504.png,Normal/masks/Normal-4504.png,True,False,299,299,L,107.7,db32949b4ded45a8a28cc2c08e4bf08b
Normal-4505.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4505.png,Normal/images/Normal-4505.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4505.png,Normal/masks/Normal-4505.png,True,False,299,299,L,155.66,1ca1b49eaa286bad11607152493676a4
Normal-4506.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4506.png,Normal/images/Normal-4506.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4506.png,Normal/masks/Normal-4506.png,True,False,299,299,L,136.08,3ab6656d40836938fdce2817e8b0f246
Normal-4507.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4507.png,Normal/images/Normal-4507.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4507.png,Normal/masks/Normal-4507.png,True,False,299,299,L,96.75,d90c62b5bebdf289c0481a0c0a84a738
Normal-4508.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4508.png,Normal/images/Normal-4508.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4508.png,Normal/masks/Normal-4508.png,True,False,299,299,L,157.28,8184683ecf30526c90359fe51cfa168f
Normal-4509.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4509.png,Normal/images/Normal-4509.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4509.png,Normal/masks/Normal-4509.png,True,False,299,299,L,165.28,bba210b47f89f350c6f51ec6056ace31
Normal-451.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-451.png,Normal/images/Normal-451.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-451.png,Normal/masks/Normal-451.png,True,False,299,299,L,128.94,829303aebc95a085514aeb55520753cb
Normal-4510.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4510.png,Normal/images/Normal-4510.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4510.png,Normal/masks/Normal-4510.png,True,False,299,299,L,153.61,278e125efbab4c70edd2021e35b058da
Normal-4511.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4511.png,Normal/images/Normal-4511.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4511.png,Normal/masks/Normal-4511.png,True,False,299,299,L,115.74,022c6d09f9d6aa93c081098499e93969
Normal-4512.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4512.png,Normal/images/Normal-4512.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4512.png,Normal/masks/Normal-4512.png,True,False,299,299,L,150.81,470c0ea6dba6f362fa81cd7e0caa4f30
Normal-4513.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4513.png,Normal/images/Normal-4513.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4513.png,Normal/masks/Normal-4513.png,True,False,299,299,L,106.3,d49c5515c338a0b61197bd2286d016cc
Normal-4514.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4514.png,Normal/images/Normal-4514.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4514.png,Normal/masks/Normal-4514.png,True,False,299,299,L,108.31,b901ad51796b26f1a2ce6bcb332f59e8
Normal-4515.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4515.png,Normal/images/Normal-4515.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4515.png,Normal/masks/Normal-4515.png,True,False,299,299,L,119.6,fce97cbe7af5962f3e6687e5899d6096
Normal-4516.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4516.png,Normal/images/Normal-4516.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4516.png,Normal/masks/Normal-4516.png,True,False,299,299,L,108.0,d0607bb81f1e1acb09ad3f8d8be3bca4
Normal-4517.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4517.png,Normal/images/Normal-4517.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4517.png,Normal/masks/Normal-4517.png,True,False,299,299,L,126.5,d231c75f30ff9c6a5ee15490e0ed9d7e
Normal-4518.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4518.png,Normal/images/Normal-4518.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4518.png,Normal/masks/Normal-4518.png,True,False,299,299,L,128.95,e41508daf9724bdbfffd6109a8ac4f9e
Normal-4519.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4519.png,Normal/images/Normal-4519.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4519.png,Normal/masks/Normal-4519.png,True,False,299,299,L,92.49,08f07ce977c232dbed4915279abcd320
Normal-452.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-452.png,Normal/images/Normal-452.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-452.png,Normal/masks/Normal-452.png,True,False,299,299,L,118.32,0224c97e381e9eb03353981e034fc3b8
Normal-4520.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4520.png,Normal/images/Normal-4520.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4520.png,Normal/masks/Normal-4520.png,True,False,299,299,L,134.79,d535e09202585c31f0968da01ec94847
Normal-4521.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4521.png,Normal/images/Normal-4521.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4521.png,Normal/masks/Normal-4521.png,True,False,299,299,L,190.94,889b3dcdb9d340699cc25f4b038fc284
Normal-4522.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4522.png,Normal/images/Normal-4522.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4522.png,Normal/masks/Normal-4522.png,True,False,299,299,L,155.36,20d880edb4492326dec5afcbba1f9af0
Normal-4523.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4523.png,Normal/images/Normal-4523.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4523.png,Normal/masks/Normal-4523.png,True,False,299,299,L,163.0,d987a36f3652e3f068da595b14df07cd
Normal-4524.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4524.png,Normal/images/Normal-4524.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4524.png,Normal/masks/Normal-4524.png,True,False,299,299,L,111.55,18e19af14d15cd8f1f2485d00be39753
Normal-4525.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4525.png,Normal/images/Normal-4525.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4525.png,Normal/masks/Normal-4525.png,True,False,299,299,L,98.27,fb079a3630e7ed120452eb81e04d666e
Normal-4526.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4526.png,Normal/images/Normal-4526.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4526.png,Normal/masks/Normal-4526.png,True,False,299,299,L,109.98,9acfdfeaa94786b16a452f2413a42199
Normal-4527.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4527.png,Normal/images/Normal-4527.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4527.png,Normal/masks/Normal-4527.png,True,False,299,299,L,156.15,e759a615af9b76a89f328eba00386e2b
Normal-4528.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4528.png,Normal/images/Normal-4528.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4528.png,Normal/masks/Normal-4528.png,True,False,299,299,L,114.11,c77877a9ff4131f9a0f617ee7f0be848
Normal-4529.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4529.png,Normal/images/Normal-4529.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4529.png,Normal/masks/Normal-4529.png,True,False,299,299,L,166.66,84b64e5a473edc736108c014a6ced4a9
Normal-453.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-453.png,Normal/images/Normal-453.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-453.png,Normal/masks/Normal-453.png,True,False,299,299,L,127.05,31ac393e87ceada342c4cc5514b3837b
Normal-4530.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4530.png,Normal/images/Normal-4530.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4530.png,Normal/masks/Normal-4530.png,True,False,299,299,L,132.97,3fb401ebd6397185f3fee7f79f62a1b8
Normal-4531.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4531.png,Normal/images/Normal-4531.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4531.png,Normal/masks/Normal-4531.png,True,False,299,299,L,126.27,0dcbb7e92518ce69658817ec5f5224b6
Normal-4532.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4532.png,Normal/images/Normal-4532.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4532.png,Normal/masks/Normal-4532.png,True,False,299,299,L,129.05,ed7c6ccdb62c7b886b91eee1b905f178
Normal-4533.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4533.png,Normal/images/Normal-4533.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4533.png,Normal/masks/Normal-4533.png,True,False,299,299,L,114.26,0165b464e67a4cb1049b2489abac0098
Normal-4534.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4534.png,Normal/images/Normal-4534.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4534.png,Normal/masks/Normal-4534.png,True,False,299,299,L,159.54,b404e7e490405b7a3af3b9f0b3394c15
Normal-4535.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4535.png,Normal/images/Normal-4535.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4535.png,Normal/masks/Normal-4535.png,True,False,299,299,L,116.85,31853efd0c18e3fcbeba635699e1ed7f
Normal-4536.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4536.png,Normal/images/Normal-4536.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4536.png,Normal/masks/Normal-4536.png,True,False,299,299,L,115.23,a4496e2d83c1c16b7100d35884e3a0a5
Normal-4537.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4537.png,Normal/images/Normal-4537.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4537.png,Normal/masks/Normal-4537.png,True,False,299,299,L,164.96,f0f4bbc1a913dee2e61b8d1fdfcb616d
Normal-4538.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4538.png,Normal/images/Normal-4538.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4538.png,Normal/masks/Normal-4538.png,True,False,299,299,L,127.72,e72a3763429111aee405a1d5fd97cc03
Normal-4539.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4539.png,Normal/images/Normal-4539.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4539.png,Normal/masks/Normal-4539.png,True,False,299,299,L,175.21,62eec28c4f479bdd97071ca30b478d0f
Normal-454.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-454.png,Normal/images/Normal-454.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-454.png,Normal/masks/Normal-454.png,True,False,299,299,L,120.3,e2ed954ff24822e058c66f786ab56684
Normal-4540.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4540.png,Normal/images/Normal-4540.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4540.png,Normal/masks/Normal-4540.png,True,False,299,299,L,182.0,34c19900ff0edb8a3490bbdee9ea68b7
Normal-4541.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4541.png,Normal/images/Normal-4541.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4541.png,Normal/masks/Normal-4541.png,True,False,299,299,L,161.39,fe877f904802c222dc263cabc096f06d
Normal-4542.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4542.png,Normal/images/Normal-4542.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4542.png,Normal/masks/Normal-4542.png,True,False,299,299,L,113.95,a2d8f04b9364236c2c1c9a882dbbb031
Normal-4543.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4543.png,Normal/images/Normal-4543.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4543.png,Normal/masks/Normal-4543.png,True,False,299,299,L,107.84,f3b8ff71e3b0acc921300095a536fe14
Normal-4544.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4544.png,Normal/images/Normal-4544.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4544.png,Normal/masks/Normal-4544.png,True,False,299,299,L,114.69,d5f79b66b29b7c15c1f34f30c6834974
Normal-4545.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4545.png,Normal/images/Normal-4545.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4545.png,Normal/masks/Normal-4545.png,True,False,299,299,L,141.51,e76abdfd3fb92b53422563f2cd6a3139
Normal-4546.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4546.png,Normal/images/Normal-4546.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4546.png,Normal/masks/Normal-4546.png,True,False,299,299,L,178.22,1347b3b667eb290bda7c2b91e04bb997
Normal-4547.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4547.png,Normal/images/Normal-4547.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4547.png,Normal/masks/Normal-4547.png,True,False,299,299,L,125.81,e4aaf94012ab20d70749e87923c4d234
Normal-4548.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4548.png,Normal/images/Normal-4548.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4548.png,Normal/masks/Normal-4548.png,True,False,299,299,L,131.61,f1c7e4d00640b42a45211394636b482d
Normal-4549.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4549.png,Normal/images/Normal-4549.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4549.png,Normal/masks/Normal-4549.png,True,False,299,299,L,114.35,6f7eedf6e3ad46ad41ba165856c5363e
Normal-455.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-455.png,Normal/images/Normal-455.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-455.png,Normal/masks/Normal-455.png,True,False,299,299,L,125.74,6f34bb34a0c15fc479cff0ff52d84b42
Normal-4550.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4550.png,Normal/images/Normal-4550.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4550.png,Normal/masks/Normal-4550.png,True,False,299,299,L,166.03,2740de7361c96d1a3fe464a894e28438
Normal-4551.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4551.png,Normal/images/Normal-4551.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4551.png,Normal/masks/Normal-4551.png,True,False,299,299,L,164.79,df6da7051bb5242c4cd690122086bc22
Normal-4552.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4552.png,Normal/images/Normal-4552.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4552.png,Normal/masks/Normal-4552.png,True,False,299,299,L,98.73,3468ba6fcb63e6a7e7114fc38d4f6741
Normal-4553.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4553.png,Normal/images/Normal-4553.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4553.png,Normal/masks/Normal-4553.png,True,False,299,299,L,119.51,0158eaddab595d2ad814232b05799d42
Normal-4554.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4554.png,Normal/images/Normal-4554.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4554.png,Normal/masks/Normal-4554.png,True,False,299,299,L,155.17,b752e3fb2fde80fb218e9ce59b88d95c
Normal-4555.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4555.png,Normal/images/Normal-4555.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4555.png,Normal/masks/Normal-4555.png,True,False,299,299,L,101.41,535000e8eb9d39b8ac2e9f0da2a9de8e
Normal-4556.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4556.png,Normal/images/Normal-4556.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4556.png,Normal/masks/Normal-4556.png,True,False,299,299,L,121.61,64c5e57e8c804d635fe508d0d8abb111
Normal-4557.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4557.png,Normal/images/Normal-4557.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4557.png,Normal/masks/Normal-4557.png,True,False,299,299,L,160.41,8d15e78d5a323251baf66200dc74e368
Normal-4558.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4558.png,Normal/images/Normal-4558.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4558.png,Normal/masks/Normal-4558.png,True,False,299,299,L,124.14,d4c35f61327332376d652dbec9450b2c
Normal-4559.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4559.png,Normal/images/Normal-4559.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4559.png,Normal/masks/Normal-4559.png,True,False,299,299,L,170.59,b187c27dc1e439e6ffacc7667dab5223
Normal-456.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-456.png,Normal/images/Normal-456.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-456.png,Normal/masks/Normal-456.png,True,False,299,299,L,133.24,02a9d46962d8a5a30f26674cbbbb2e82
Normal-4560.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4560.png,Normal/images/Normal-4560.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4560.png,Normal/masks/Normal-4560.png,True,False,299,299,L,140.4,902354ec3388d405a38375141f96d90a
Normal-4561.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4561.png,Normal/images/Normal-4561.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4561.png,Normal/masks/Normal-4561.png,True,False,299,299,L,138.12,6c8a7fd4f300febcaf8f642adb2d0cca
Normal-4562.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4562.png,Normal/images/Normal-4562.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4562.png,Normal/masks/Normal-4562.png,True,False,299,299,L,116.95,295fc4fe0f81b75c5b2e4d49aec82845
Normal-4563.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4563.png,Normal/images/Normal-4563.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4563.png,Normal/masks/Normal-4563.png,True,False,299,299,L,87.26,7e8e898e8b782179aee3026d25d4fd8c
Normal-4564.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4564.png,Normal/images/Normal-4564.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4564.png,Normal/masks/Normal-4564.png,True,False,299,299,L,115.79,fa08ac314bc50a37d5a0b7314a1e8e49
Normal-4565.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4565.png,Normal/images/Normal-4565.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4565.png,Normal/masks/Normal-4565.png,True,False,299,299,L,129.18,128b9895c078d952b03c03058fdc5eb3
Normal-4566.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4566.png,Normal/images/Normal-4566.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4566.png,Normal/masks/Normal-4566.png,True,False,299,299,L,130.47,866d804a99fba04c27c65307e820b8f8
Normal-4567.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4567.png,Normal/images/Normal-4567.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4567.png,Normal/masks/Normal-4567.png,True,False,299,299,L,124.99,12ea22beaac292a669f4f8b09174da4c
Normal-4568.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4568.png,Normal/images/Normal-4568.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4568.png,Normal/masks/Normal-4568.png,True,False,299,299,L,141.63,10c7b9ae2f834655216856c4007bc6bb
Normal-4569.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4569.png,Normal/images/Normal-4569.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4569.png,Normal/masks/Normal-4569.png,True,False,299,299,L,159.9,593e2f92b8154744538c38f596aae5e4
Normal-457.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-457.png,Normal/images/Normal-457.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-457.png,Normal/masks/Normal-457.png,True,False,299,299,L,120.49,6d6cdc7ec1b132f091d2aeb98317fb26
Normal-4570.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4570.png,Normal/images/Normal-4570.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4570.png,Normal/masks/Normal-4570.png,True,False,299,299,L,156.61,70a0b5fa5dcedd09146b7a1e93eaa7d6
Normal-4571.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4571.png,Normal/images/Normal-4571.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4571.png,Normal/masks/Normal-4571.png,True,False,299,299,L,151.21,f41e660ea3639e2efab1bd41f687c94b
Normal-4572.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4572.png,Normal/images/Normal-4572.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4572.png,Normal/masks/Normal-4572.png,True,False,299,299,L,151.81,efd2f1fb62a6c6f8061245df1fd54567
Normal-4573.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4573.png,Normal/images/Normal-4573.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4573.png,Normal/masks/Normal-4573.png,True,False,299,299,L,96.46,664d43dc7f492ddf7638a3dfe5cd636e
Normal-4574.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4574.png,Normal/images/Normal-4574.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4574.png,Normal/masks/Normal-4574.png,True,False,299,299,L,119.8,8630be07a0a4311e053d444a1817ce85
Normal-4575.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4575.png,Normal/images/Normal-4575.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4575.png,Normal/masks/Normal-4575.png,True,False,299,299,L,160.24,86525944475d530a02553b524ad74bbf
Normal-4576.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4576.png,Normal/images/Normal-4576.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4576.png,Normal/masks/Normal-4576.png,True,False,299,299,L,127.31,6df613d65105821a88f6d92e78245e3c
Normal-4577.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4577.png,Normal/images/Normal-4577.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4577.png,Normal/masks/Normal-4577.png,True,False,299,299,L,139.27,8674aafb8654228d3f7dc4786c0b997e
Normal-4578.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4578.png,Normal/images/Normal-4578.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4578.png,Normal/masks/Normal-4578.png,True,False,299,299,L,111.88,8989852891b4d5d6660b67247a8c38a6
Normal-4579.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4579.png,Normal/images/Normal-4579.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4579.png,Normal/masks/Normal-4579.png,True,False,299,299,L,174.08,5727b5fdfb57d891d46a46a0b6c87ba1
Normal-458.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-458.png,Normal/images/Normal-458.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-458.png,Normal/masks/Normal-458.png,True,False,299,299,L,130.78,0daf78669c7c715f41fb55e2eb4fd2e9
Normal-4580.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4580.png,Normal/images/Normal-4580.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4580.png,Normal/masks/Normal-4580.png,True,False,299,299,L,120.49,6786b49b6e14a6f42f9d76f74ee1a797
Normal-4581.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4581.png,Normal/images/Normal-4581.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4581.png,Normal/masks/Normal-4581.png,True,False,299,299,L,118.19,d9dc8007aecc042508f18a97f57bfc33
Normal-4582.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4582.png,Normal/images/Normal-4582.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4582.png,Normal/masks/Normal-4582.png,True,False,299,299,L,169.11,83e09848da792846c70bf49cd488af48
Normal-4583.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4583.png,Normal/images/Normal-4583.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4583.png,Normal/masks/Normal-4583.png,True,False,299,299,L,131.89,dc5bca5ef5363a534bba64b87c001918
Normal-4584.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4584.png,Normal/images/Normal-4584.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4584.png,Normal/masks/Normal-4584.png,True,False,299,299,L,172.58,ae5433756ff57caac8d9a9cc52fac9e0
Normal-4585.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4585.png,Normal/images/Normal-4585.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4585.png,Normal/masks/Normal-4585.png,True,False,299,299,L,117.2,c7875faf9ca9b538daf33eba02dc5aa2
Normal-4586.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4586.png,Normal/images/Normal-4586.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4586.png,Normal/masks/Normal-4586.png,True,False,299,299,L,120.07,60a52be6ce2773cc48b3e7c56fe92795
Normal-4587.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4587.png,Normal/images/Normal-4587.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4587.png,Normal/masks/Normal-4587.png,True,False,299,299,L,109.51,994c0ae70e2d969e249accd141edafbf
Normal-4588.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4588.png,Normal/images/Normal-4588.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4588.png,Normal/masks/Normal-4588.png,True,False,299,299,L,175.72,74ceada0631076dd927f03c1ba736753
Normal-4589.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4589.png,Normal/images/Normal-4589.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4589.png,Normal/masks/Normal-4589.png,True,False,299,299,L,128.55,09273f8c0825708f681b593e4360a82e
Normal-459.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-459.png,Normal/images/Normal-459.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-459.png,Normal/masks/Normal-459.png,True,False,299,299,L,118.43,9c292310a024f65145483b639f5a8a03
Normal-4590.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4590.png,Normal/images/Normal-4590.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4590.png,Normal/masks/Normal-4590.png,True,False,299,299,L,120.71,fc35ddd014f717fd2344e9f2b7c35b10
Normal-4591.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4591.png,Normal/images/Normal-4591.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4591.png,Normal/masks/Normal-4591.png,True,False,299,299,L,122.7,a95cab4dc051d71125a3644bc559cca2
Normal-4592.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4592.png,Normal/images/Normal-4592.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4592.png,Normal/masks/Normal-4592.png,True,False,299,299,L,109.73,a1353d8f5880a2dc6dcc4042b4848b58
Normal-4593.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4593.png,Normal/images/Normal-4593.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4593.png,Normal/masks/Normal-4593.png,True,False,299,299,L,125.02,c2b2332711fe675c5cea75ba1f243e49
Normal-4594.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4594.png,Normal/images/Normal-4594.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4594.png,Normal/masks/Normal-4594.png,True,False,299,299,L,140.24,0418ffe54fa85fde8d3fb92c7f218f87
Normal-4595.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4595.png,Normal/images/Normal-4595.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4595.png,Normal/masks/Normal-4595.png,True,False,299,299,L,135.57,f2a371d425f652dcf027b41ad2ba8fb4
Normal-4596.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4596.png,Normal/images/Normal-4596.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4596.png,Normal/masks/Normal-4596.png,True,False,299,299,L,106.89,c6e6c005a1ff5c81a4492bcab1b24964
Normal-4597.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4597.png,Normal/images/Normal-4597.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4597.png,Normal/masks/Normal-4597.png,True,False,299,299,L,169.92,6440be4f5b5fd07c0a710fa9a9a177bb
Normal-4598.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4598.png,Normal/images/Normal-4598.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4598.png,Normal/masks/Normal-4598.png,True,False,299,299,L,156.44,4bea3ffa1defbb956c549a92da55f456
Normal-4599.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4599.png,Normal/images/Normal-4599.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4599.png,Normal/masks/Normal-4599.png,True,False,299,299,L,108.31,6c6b16a4cefa85e98b7f528fa8677971
Normal-46.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-46.png,Normal/images/Normal-46.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-46.png,Normal/masks/Normal-46.png,True,False,299,299,L,129.08,a6ddf04a961d3ac83532ce3b9b7a9e45
Normal-460.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-460.png,Normal/images/Normal-460.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-460.png,Normal/masks/Normal-460.png,True,False,299,299,L,129.3,d3d5fb66a0dd645afcd7ff905f0baefb
Normal-4600.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4600.png,Normal/images/Normal-4600.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4600.png,Normal/masks/Normal-4600.png,True,False,299,299,L,120.81,6921a1193f5b1660a0613974a9ee3a73
Normal-4601.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4601.png,Normal/images/Normal-4601.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4601.png,Normal/masks/Normal-4601.png,True,False,299,299,L,113.88,5ecdd8915f4405869ce0d1ca8bd4f7ec
Normal-4602.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4602.png,Normal/images/Normal-4602.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4602.png,Normal/masks/Normal-4602.png,True,False,299,299,L,125.31,58996dd43f067a593e2909b49d0f2e7f
Normal-4603.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4603.png,Normal/images/Normal-4603.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4603.png,Normal/masks/Normal-4603.png,True,False,299,299,L,121.76,85e2f29ea2c8acf9d3b23db466b9160b
Normal-4604.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4604.png,Normal/images/Normal-4604.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4604.png,Normal/masks/Normal-4604.png,True,False,299,299,L,104.07,2504e8acbb16946670085cf863f65622
Normal-4605.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4605.png,Normal/images/Normal-4605.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4605.png,Normal/masks/Normal-4605.png,True,False,299,299,L,128.24,94329e0f4fa4ee7243bbc95dda258e48
Normal-4606.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4606.png,Normal/images/Normal-4606.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4606.png,Normal/masks/Normal-4606.png,True,False,299,299,L,148.0,d5bdade1f1038dbf962caf7ee6be4112
Normal-4607.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4607.png,Normal/images/Normal-4607.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4607.png,Normal/masks/Normal-4607.png,True,False,299,299,L,119.71,95400b353d4bfb87c91c2476d1effa37
Normal-4608.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4608.png,Normal/images/Normal-4608.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4608.png,Normal/masks/Normal-4608.png,True,False,299,299,L,133.8,aa5ec1157b1f51d9ce7457a3b05273d5
Normal-4609.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4609.png,Normal/images/Normal-4609.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4609.png,Normal/masks/Normal-4609.png,True,False,299,299,L,85.36,9e81f7c0f4e761fc8230a6b9b71636aa
Normal-461.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-461.png,Normal/images/Normal-461.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-461.png,Normal/masks/Normal-461.png,True,False,299,299,L,132.49,5d5d8cbf3fc133af91f09bde7e026aee
Normal-4610.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4610.png,Normal/images/Normal-4610.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4610.png,Normal/masks/Normal-4610.png,True,False,299,299,L,127.08,17f3fa1036cec2e2da7bdc9204145307
Normal-4611.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4611.png,Normal/images/Normal-4611.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4611.png,Normal/masks/Normal-4611.png,True,False,299,299,L,145.48,475e7b45cc0bf3eefe8550e0df51f083
Normal-4612.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4612.png,Normal/images/Normal-4612.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4612.png,Normal/masks/Normal-4612.png,True,False,299,299,L,93.31,6a27e58911b7844c7e79a19c5aff17f7
Normal-4613.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4613.png,Normal/images/Normal-4613.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4613.png,Normal/masks/Normal-4613.png,True,False,299,299,L,165.79,3eb2aae071264055d4a9a7c4cf7ce4f5
Normal-4614.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4614.png,Normal/images/Normal-4614.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4614.png,Normal/masks/Normal-4614.png,True,False,299,299,L,110.53,cc67844167866be8f0679783eed03bc8
Normal-4615.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4615.png,Normal/images/Normal-4615.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4615.png,Normal/masks/Normal-4615.png,True,False,299,299,L,100.53,e6c26763454a2745dab78c39d322f067
Normal-4616.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4616.png,Normal/images/Normal-4616.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4616.png,Normal/masks/Normal-4616.png,True,False,299,299,L,149.44,59a8e19447ad1ea93233a1c944797a0b
Normal-4617.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4617.png,Normal/images/Normal-4617.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4617.png,Normal/masks/Normal-4617.png,True,False,299,299,L,182.91,aa10c64dceb2f6aefc48cdef7ed1fda4
Normal-4618.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4618.png,Normal/images/Normal-4618.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4618.png,Normal/masks/Normal-4618.png,True,False,299,299,L,107.45,788d1a051d1665c4c50788d6a66192d9
Normal-4619.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4619.png,Normal/images/Normal-4619.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4619.png,Normal/masks/Normal-4619.png,True,False,299,299,L,108.1,89abd54411497423d8e8c9c0e58c7306
Normal-462.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-462.png,Normal/images/Normal-462.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-462.png,Normal/masks/Normal-462.png,True,False,299,299,L,119.35,2279f078d7bbbd5df041d8fd0dfc16ab
Normal-4620.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4620.png,Normal/images/Normal-4620.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4620.png,Normal/masks/Normal-4620.png,True,False,299,299,L,152.01,3c4fbbb1bd0381c3caf77df03b627f56
Normal-4621.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4621.png,Normal/images/Normal-4621.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4621.png,Normal/masks/Normal-4621.png,True,False,299,299,L,117.6,f76f7cfcce03291cd1d94426e72054b3
Normal-4622.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4622.png,Normal/images/Normal-4622.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4622.png,Normal/masks/Normal-4622.png,True,False,299,299,L,109.65,b9d738f6c6c359c53e47c19c4acff62b
Normal-4623.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4623.png,Normal/images/Normal-4623.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4623.png,Normal/masks/Normal-4623.png,True,False,299,299,L,94.94,7557d812c61f12bdfcd589c4607305a8
Normal-4624.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4624.png,Normal/images/Normal-4624.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4624.png,Normal/masks/Normal-4624.png,True,False,299,299,L,161.11,2194708b483dde39030c862862a24e3d
Normal-4625.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4625.png,Normal/images/Normal-4625.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4625.png,Normal/masks/Normal-4625.png,True,False,299,299,L,112.47,e379134d7f1af7a1d90485a704d90fc6
Normal-4626.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4626.png,Normal/images/Normal-4626.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4626.png,Normal/masks/Normal-4626.png,True,False,299,299,L,165.37,692c9e8f97d92e01c004c8150125e97b
Normal-4627.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4627.png,Normal/images/Normal-4627.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4627.png,Normal/masks/Normal-4627.png,True,False,299,299,L,61.85,5f443403bfe9125189b78296f3d429cf
Normal-4628.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4628.png,Normal/images/Normal-4628.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4628.png,Normal/masks/Normal-4628.png,True,False,299,299,L,120.28,d1eaafb0b5da8fd09434da59fbfcca5e
Normal-4629.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4629.png,Normal/images/Normal-4629.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4629.png,Normal/masks/Normal-4629.png,True,False,299,299,L,99.72,01adb2a7644e10235d2e49181eab5968
Normal-463.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-463.png,Normal/images/Normal-463.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-463.png,Normal/masks/Normal-463.png,True,False,299,299,L,137.03,8e1e8c2222ac92071cc543f6fc02ed84
Normal-4630.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4630.png,Normal/images/Normal-4630.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4630.png,Normal/masks/Normal-4630.png,True,False,299,299,L,89.12,cc942159c8e1cb58d407a6dac446498b
Normal-4631.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4631.png,Normal/images/Normal-4631.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4631.png,Normal/masks/Normal-4631.png,True,False,299,299,L,98.57,1d5c5a275abac65c198f4d7012e0fe66
Normal-4632.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4632.png,Normal/images/Normal-4632.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4632.png,Normal/masks/Normal-4632.png,True,False,299,299,L,106.25,1577ae0f181a8c072050ef1028e5ff2a
Normal-4633.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4633.png,Normal/images/Normal-4633.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4633.png,Normal/masks/Normal-4633.png,True,False,299,299,L,134.15,9090f9696d0d874d224223361fc1194c
Normal-4634.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4634.png,Normal/images/Normal-4634.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4634.png,Normal/masks/Normal-4634.png,True,False,299,299,L,104.98,f5a2f15c782bbde79d2973006a534a03
Normal-4635.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4635.png,Normal/images/Normal-4635.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4635.png,Normal/masks/Normal-4635.png,True,False,299,299,L,166.88,1f2c9a1baf94e01b99c3650ede38467e
Normal-4636.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4636.png,Normal/images/Normal-4636.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4636.png,Normal/masks/Normal-4636.png,True,False,299,299,L,103.18,fd172352a8c69c4a28717425e43a2ac4
Normal-4637.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4637.png,Normal/images/Normal-4637.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4637.png,Normal/masks/Normal-4637.png,True,False,299,299,L,144.89,30c83c7736745737208446160d4e2554
Normal-4638.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4638.png,Normal/images/Normal-4638.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4638.png,Normal/masks/Normal-4638.png,True,False,299,299,L,122.76,c328f569da5bbf3adb464630a35ac70e
Normal-4639.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4639.png,Normal/images/Normal-4639.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4639.png,Normal/masks/Normal-4639.png,True,False,299,299,L,112.19,7f834ca5b684ea71805ad64e690243c7
Normal-464.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-464.png,Normal/images/Normal-464.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-464.png,Normal/masks/Normal-464.png,True,False,299,299,L,117.67,b79badd3ca169af06585edde1c4b48eb
Normal-4640.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4640.png,Normal/images/Normal-4640.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4640.png,Normal/masks/Normal-4640.png,True,False,299,299,L,133.11,ff4888cc57644c11862e5904669296f5
Normal-4641.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4641.png,Normal/images/Normal-4641.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4641.png,Normal/masks/Normal-4641.png,True,False,299,299,L,151.06,c6a36fc8ae4625f96da163bb37361867
Normal-4642.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4642.png,Normal/images/Normal-4642.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4642.png,Normal/masks/Normal-4642.png,True,False,299,299,L,114.34,b91dd56feb90ef67e615b117b77e350a
Normal-4643.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4643.png,Normal/images/Normal-4643.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4643.png,Normal/masks/Normal-4643.png,True,False,299,299,L,122.37,9b960f1180e3fd29860c6b498921e3cb
Normal-4644.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4644.png,Normal/images/Normal-4644.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4644.png,Normal/masks/Normal-4644.png,True,False,299,299,L,140.84,cb8aa158b937aa913383b4f14aad232b
Normal-4645.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4645.png,Normal/images/Normal-4645.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4645.png,Normal/masks/Normal-4645.png,True,False,299,299,L,111.14,a25797050b9cdb14be2350c682970e4b
Normal-4646.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4646.png,Normal/images/Normal-4646.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4646.png,Normal/masks/Normal-4646.png,True,False,299,299,L,100.44,34d795b783183dd29e444d3873f19ed3
Normal-4647.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4647.png,Normal/images/Normal-4647.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4647.png,Normal/masks/Normal-4647.png,True,False,299,299,L,148.03,8f958e834a73bafe4282d96473961139
Normal-4648.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4648.png,Normal/images/Normal-4648.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4648.png,Normal/masks/Normal-4648.png,True,False,299,299,L,120.17,e10586b1c28469d96392239223a3b581
Normal-4649.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4649.png,Normal/images/Normal-4649.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4649.png,Normal/masks/Normal-4649.png,True,False,299,299,L,119.89,8d66d06eca0824204c345e2628c06c01
Normal-465.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-465.png,Normal/images/Normal-465.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-465.png,Normal/masks/Normal-465.png,True,False,299,299,L,126.46,41df25ca7db4c9feb5be099045c666dd
Normal-4650.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4650.png,Normal/images/Normal-4650.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4650.png,Normal/masks/Normal-4650.png,True,False,299,299,L,132.61,1524a361efd4e65591aa4211754ec79b
Normal-4651.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4651.png,Normal/images/Normal-4651.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4651.png,Normal/masks/Normal-4651.png,True,False,299,299,L,162.68,72e9bc8c2dff9bbffcef7de5a6cb987e
Normal-4652.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4652.png,Normal/images/Normal-4652.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4652.png,Normal/masks/Normal-4652.png,True,False,299,299,L,167.97,1f6d9fee251ed2cf6d2095206e791a6d
Normal-4653.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4653.png,Normal/images/Normal-4653.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4653.png,Normal/masks/Normal-4653.png,True,False,299,299,L,109.71,115e3ee37d2eac1b725b9e2baef50151
Normal-4654.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4654.png,Normal/images/Normal-4654.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4654.png,Normal/masks/Normal-4654.png,True,False,299,299,L,105.77,e2b502a3475ba6df0d34ecbd0ff10479
Normal-4655.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4655.png,Normal/images/Normal-4655.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4655.png,Normal/masks/Normal-4655.png,True,False,299,299,L,149.19,b2feeba200f281f79236c9224e499d3d
Normal-4656.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4656.png,Normal/images/Normal-4656.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4656.png,Normal/masks/Normal-4656.png,True,False,299,299,L,118.49,63d9b78d272b13945de251fedb947037
Normal-4657.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4657.png,Normal/images/Normal-4657.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4657.png,Normal/masks/Normal-4657.png,True,False,299,299,L,117.64,5b9cb496377f46e2d6bb1635583ca499
Normal-4658.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4658.png,Normal/images/Normal-4658.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4658.png,Normal/masks/Normal-4658.png,True,False,299,299,L,161.73,75721d3a7403f603056c78d6ca13f92b
Normal-4659.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4659.png,Normal/images/Normal-4659.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4659.png,Normal/masks/Normal-4659.png,True,False,299,299,L,125.49,b5095f1144664183758765181730390a
Normal-466.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-466.png,Normal/images/Normal-466.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-466.png,Normal/masks/Normal-466.png,True,False,299,299,L,128.74,f723d075968e4520d24524d3b1141252
Normal-4660.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4660.png,Normal/images/Normal-4660.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4660.png,Normal/masks/Normal-4660.png,True,False,299,299,L,124.81,24f4d03a722ae281f4e556ddac8c92b8
Normal-4661.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4661.png,Normal/images/Normal-4661.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4661.png,Normal/masks/Normal-4661.png,True,False,299,299,L,158.65,e039ebdc339a2b0afa6ac8544a9614e1
Normal-4662.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4662.png,Normal/images/Normal-4662.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4662.png,Normal/masks/Normal-4662.png,True,False,299,299,L,124.22,029ee926ebf4c78382d72c15bf3c6e3b
Normal-4663.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4663.png,Normal/images/Normal-4663.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4663.png,Normal/masks/Normal-4663.png,True,False,299,299,L,146.8,9846a874b64809a9c557c04a63d776e7
Normal-4664.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4664.png,Normal/images/Normal-4664.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4664.png,Normal/masks/Normal-4664.png,True,False,299,299,L,100.88,56561b58d0a62731f1617c6a89d0e489
Normal-4665.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4665.png,Normal/images/Normal-4665.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4665.png,Normal/masks/Normal-4665.png,True,False,299,299,L,113.65,5ad2fcb45179c410efd577070f75c524
Normal-4666.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4666.png,Normal/images/Normal-4666.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4666.png,Normal/masks/Normal-4666.png,True,False,299,299,L,102.62,d63a8246a7ff54f8e961f6a0ccbda95b
Normal-4667.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4667.png,Normal/images/Normal-4667.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4667.png,Normal/masks/Normal-4667.png,True,False,299,299,L,116.8,16f8ece9dfa59b05d353e53f1b433add
Normal-4668.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4668.png,Normal/images/Normal-4668.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4668.png,Normal/masks/Normal-4668.png,True,False,299,299,L,156.65,d440620b8789ed25dcb06202f026cd41
Normal-4669.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4669.png,Normal/images/Normal-4669.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4669.png,Normal/masks/Normal-4669.png,True,False,299,299,L,139.09,3c9fd9a1464c4f0ccab1e4de99c57f9e
Normal-467.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-467.png,Normal/images/Normal-467.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-467.png,Normal/masks/Normal-467.png,True,False,299,299,L,118.5,df4c03666ee6284aa3393a5b55ac80a4
Normal-4670.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4670.png,Normal/images/Normal-4670.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4670.png,Normal/masks/Normal-4670.png,True,False,299,299,L,137.44,6032e37d3a01c788c8de1b4a1774a7b9
Normal-4671.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4671.png,Normal/images/Normal-4671.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4671.png,Normal/masks/Normal-4671.png,True,False,299,299,L,176.35,c989d0c40626f571b411bd50dbe19833
Normal-4672.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4672.png,Normal/images/Normal-4672.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4672.png,Normal/masks/Normal-4672.png,True,False,299,299,L,116.49,0d1a3ad6fcf52b23927ef390cc1e1b15
Normal-4673.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4673.png,Normal/images/Normal-4673.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4673.png,Normal/masks/Normal-4673.png,True,False,299,299,L,172.32,203643783e51fb9c8022ff89ea99a4c7
Normal-4674.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4674.png,Normal/images/Normal-4674.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4674.png,Normal/masks/Normal-4674.png,True,False,299,299,L,148.24,dea8ef243e0304bece3671b94820ca54
Normal-4675.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4675.png,Normal/images/Normal-4675.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4675.png,Normal/masks/Normal-4675.png,True,False,299,299,L,123.12,f871d30dcbda56a299afa04f82267665
Normal-4676.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4676.png,Normal/images/Normal-4676.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4676.png,Normal/masks/Normal-4676.png,True,False,299,299,L,127.36,9babf1f646cdea4becc5c5859f7bba3b
Normal-4677.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4677.png,Normal/images/Normal-4677.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4677.png,Normal/masks/Normal-4677.png,True,False,299,299,L,147.21,baca847f11288776b2bca5268c770727
Normal-4678.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4678.png,Normal/images/Normal-4678.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4678.png,Normal/masks/Normal-4678.png,True,False,299,299,L,115.89,b70ba179cbd76404c243c417c39fa877
Normal-4679.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4679.png,Normal/images/Normal-4679.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4679.png,Normal/masks/Normal-4679.png,True,False,299,299,L,123.48,8dd2f85641f59091e3646cbecc3b2ff8
Normal-468.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-468.png,Normal/images/Normal-468.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-468.png,Normal/masks/Normal-468.png,True,False,299,299,L,134.89,8b1661ec768a8683177117002c375aed
Normal-4680.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4680.png,Normal/images/Normal-4680.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4680.png,Normal/masks/Normal-4680.png,True,False,299,299,L,106.46,bcdaef3a05fa3ef37f5a4f770ad2860e
Normal-4681.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4681.png,Normal/images/Normal-4681.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4681.png,Normal/masks/Normal-4681.png,True,False,299,299,L,132.39,24abb33b2bdf0d10d797b39ac724a7a3
Normal-4682.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4682.png,Normal/images/Normal-4682.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4682.png,Normal/masks/Normal-4682.png,True,False,299,299,L,158.09,05280dd3fe63a4a2d90e431291815ff9
Normal-4683.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4683.png,Normal/images/Normal-4683.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4683.png,Normal/masks/Normal-4683.png,True,False,299,299,L,123.5,38a7aa89f94b97ea40bb832383f35f1d
Normal-4684.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4684.png,Normal/images/Normal-4684.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4684.png,Normal/masks/Normal-4684.png,True,False,299,299,L,109.21,adcf5ba3cb1fe20cdf7b574d88fd066c
Normal-4685.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4685.png,Normal/images/Normal-4685.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4685.png,Normal/masks/Normal-4685.png,True,False,299,299,L,106.79,1880b4c908828e4793e7b66603ff51b5
Normal-4686.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4686.png,Normal/images/Normal-4686.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4686.png,Normal/masks/Normal-4686.png,True,False,299,299,L,173.76,485be8b8baa9d61e9cda73694599b6e6
Normal-4687.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4687.png,Normal/images/Normal-4687.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4687.png,Normal/masks/Normal-4687.png,True,False,299,299,L,126.53,662f20dbea62f1a3066f671a809d5cde
Normal-4688.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4688.png,Normal/images/Normal-4688.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4688.png,Normal/masks/Normal-4688.png,True,False,299,299,L,115.73,a265b27e69cf61bc3be9e2b1d71ea922
Normal-4689.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4689.png,Normal/images/Normal-4689.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4689.png,Normal/masks/Normal-4689.png,True,False,299,299,L,112.74,caba93e62c5534e07506f419bc1a3622
Normal-469.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-469.png,Normal/images/Normal-469.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-469.png,Normal/masks/Normal-469.png,True,False,299,299,L,119.14,07413307da02ca087a5ee9dfa16535fa
Normal-4690.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4690.png,Normal/images/Normal-4690.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4690.png,Normal/masks/Normal-4690.png,True,False,299,299,L,113.54,2c68692cc60c2a4be3cf6b815f30dada
Normal-4691.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4691.png,Normal/images/Normal-4691.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4691.png,Normal/masks/Normal-4691.png,True,False,299,299,L,126.97,dc01da5ce2e30b052f57be1c238cb477
Normal-4692.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4692.png,Normal/images/Normal-4692.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4692.png,Normal/masks/Normal-4692.png,True,False,299,299,L,117.37,20773f9cf5c35fac751ff9b212e78256
Normal-4693.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4693.png,Normal/images/Normal-4693.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4693.png,Normal/masks/Normal-4693.png,True,False,299,299,L,180.37,51042b0a3caade004d9cbf590ceb7128
Normal-4694.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4694.png,Normal/images/Normal-4694.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4694.png,Normal/masks/Normal-4694.png,True,False,299,299,L,113.19,431d578074d95c79f2798e487a293a9c
Normal-4695.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4695.png,Normal/images/Normal-4695.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4695.png,Normal/masks/Normal-4695.png,True,False,299,299,L,96.62,4fa770d3d9339179a8503f3aac51bc72
Normal-4696.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4696.png,Normal/images/Normal-4696.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4696.png,Normal/masks/Normal-4696.png,True,False,299,299,L,106.52,dd465085baa71d79cbb5376a6243d868
Normal-4697.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4697.png,Normal/images/Normal-4697.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4697.png,Normal/masks/Normal-4697.png,True,False,299,299,L,107.64,563b080a30f993797ddc28d6329d1f91
Normal-4698.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4698.png,Normal/images/Normal-4698.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4698.png,Normal/masks/Normal-4698.png,True,False,299,299,L,137.17,76de0c20c8648f047480f34e4a588b1f
Normal-4699.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4699.png,Normal/images/Normal-4699.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4699.png,Normal/masks/Normal-4699.png,True,False,299,299,L,160.2,523fccd2c04ccc21cbf80d1f17d4201d
Normal-47.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-47.png,Normal/images/Normal-47.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-47.png,Normal/masks/Normal-47.png,True,False,299,299,L,113.89,3d2a50abb88d627ce92789648e7a40a6
Normal-470.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-470.png,Normal/images/Normal-470.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-470.png,Normal/masks/Normal-470.png,True,False,299,299,L,125.4,7229b31cdf4935ff515905e6a7698049
Normal-4700.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4700.png,Normal/images/Normal-4700.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4700.png,Normal/masks/Normal-4700.png,True,False,299,299,L,138.06,8e435149557cd649176c34a2b7a6990a
Normal-4701.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4701.png,Normal/images/Normal-4701.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4701.png,Normal/masks/Normal-4701.png,True,False,299,299,L,115.14,9cd701f2fca51b6a847270e266b9ad0f
Normal-4702.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4702.png,Normal/images/Normal-4702.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4702.png,Normal/masks/Normal-4702.png,True,False,299,299,L,133.3,af3ac6788e74c20fdddf013094869887
Normal-4703.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4703.png,Normal/images/Normal-4703.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4703.png,Normal/masks/Normal-4703.png,True,False,299,299,L,169.86,091252a6aefb9b2321c8a85d66adbffc
Normal-4704.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4704.png,Normal/images/Normal-4704.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4704.png,Normal/masks/Normal-4704.png,True,False,299,299,L,125.62,295127a77ac9d50986e1d881e055ec96
Normal-4705.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4705.png,Normal/images/Normal-4705.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4705.png,Normal/masks/Normal-4705.png,True,False,299,299,L,106.14,41d9b02ca31fad4fdbd69b664df8a9bc
Normal-4706.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4706.png,Normal/images/Normal-4706.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4706.png,Normal/masks/Normal-4706.png,True,False,299,299,L,172.66,50e52b9a290b380a9c7113a22aaaece9
Normal-4707.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4707.png,Normal/images/Normal-4707.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4707.png,Normal/masks/Normal-4707.png,True,False,299,299,L,120.88,d617b6d1879254037533ec4728e490aa
Normal-4708.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4708.png,Normal/images/Normal-4708.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4708.png,Normal/masks/Normal-4708.png,True,False,299,299,L,120.83,dab32cd88268346238a3c6ccd0634e26
Normal-4709.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4709.png,Normal/images/Normal-4709.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4709.png,Normal/masks/Normal-4709.png,True,False,299,299,L,166.27,a70b26664ea67bce0bf7f8ac1d7c7788
Normal-471.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-471.png,Normal/images/Normal-471.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-471.png,Normal/masks/Normal-471.png,True,False,299,299,L,137.96,fe61662d658239df1b6e63769cbfd6c2
Normal-4710.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4710.png,Normal/images/Normal-4710.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4710.png,Normal/masks/Normal-4710.png,True,False,299,299,L,145.85,8808abcb4e2de352b8e20637d4a6248d
Normal-4711.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4711.png,Normal/images/Normal-4711.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4711.png,Normal/masks/Normal-4711.png,True,False,299,299,L,109.8,758cbf081b1b24a7109a14c40a35e492
Normal-4712.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4712.png,Normal/images/Normal-4712.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4712.png,Normal/masks/Normal-4712.png,True,False,299,299,L,141.34,254a18d9d2e3f5766ed993a5c8dc7812
Normal-4713.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4713.png,Normal/images/Normal-4713.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4713.png,Normal/masks/Normal-4713.png,True,False,299,299,L,111.28,4c8744fa6325ae9174ed85f8b66f527c
Normal-4714.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4714.png,Normal/images/Normal-4714.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4714.png,Normal/masks/Normal-4714.png,True,False,299,299,L,124.7,6dccb91747ee22cc609dbac7c53537de
Normal-4715.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4715.png,Normal/images/Normal-4715.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4715.png,Normal/masks/Normal-4715.png,True,False,299,299,L,139.37,aa9afc238aa01da50c5a1bf308d50be0
Normal-4716.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4716.png,Normal/images/Normal-4716.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4716.png,Normal/masks/Normal-4716.png,True,False,299,299,L,117.48,328938347862823d65a9a190690bbef0
Normal-4717.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4717.png,Normal/images/Normal-4717.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4717.png,Normal/masks/Normal-4717.png,True,False,299,299,L,166.26,2f3c0f3dc715a4a26c3ff0775c998f6d
Normal-4718.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4718.png,Normal/images/Normal-4718.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4718.png,Normal/masks/Normal-4718.png,True,False,299,299,L,130.08,052c41e8eb08a0171118cec1cb6fdb0d
Normal-4719.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4719.png,Normal/images/Normal-4719.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4719.png,Normal/masks/Normal-4719.png,True,False,299,299,L,135.66,a158d73490255f41f3260eeb03d82c6f
Normal-472.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-472.png,Normal/images/Normal-472.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-472.png,Normal/masks/Normal-472.png,True,False,299,299,L,117.01,0bc6959ba53e608045e08c1a800a794f
Normal-4720.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4720.png,Normal/images/Normal-4720.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4720.png,Normal/masks/Normal-4720.png,True,False,299,299,L,137.59,aa02fdabf315d3bb0e1a361a72e5eb5d
Normal-4721.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4721.png,Normal/images/Normal-4721.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4721.png,Normal/masks/Normal-4721.png,True,False,299,299,L,114.9,c9cbb42e5162984678bab43b51fc084a
Normal-4722.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4722.png,Normal/images/Normal-4722.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4722.png,Normal/masks/Normal-4722.png,True,False,299,299,L,103.25,71f5ed2c2f89da85403a2d92e856c67f
Normal-4723.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4723.png,Normal/images/Normal-4723.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4723.png,Normal/masks/Normal-4723.png,True,False,299,299,L,137.64,bbca6da7859f6a8af88693eb20294a4b
Normal-4724.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4724.png,Normal/images/Normal-4724.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4724.png,Normal/masks/Normal-4724.png,True,False,299,299,L,131.91,435e70af18b195873580da42093d836e
Normal-4725.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4725.png,Normal/images/Normal-4725.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4725.png,Normal/masks/Normal-4725.png,True,False,299,299,L,157.01,8090c383983ed5a5493214b6e8bc4426
Normal-4726.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4726.png,Normal/images/Normal-4726.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4726.png,Normal/masks/Normal-4726.png,True,False,299,299,L,119.32,f5084693c06643afcbade7ce3af0c8c8
Normal-4727.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4727.png,Normal/images/Normal-4727.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4727.png,Normal/masks/Normal-4727.png,True,False,299,299,L,111.14,a3819bf109e05ce4c40a7f39d3232318
Normal-4728.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4728.png,Normal/images/Normal-4728.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4728.png,Normal/masks/Normal-4728.png,True,False,299,299,L,156.59,a758bdf6b1b06811757571548f8890e0
Normal-4729.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4729.png,Normal/images/Normal-4729.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4729.png,Normal/masks/Normal-4729.png,True,False,299,299,L,132.35,5f685280eacfd4103ab265c0dcf3154a
Normal-473.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-473.png,Normal/images/Normal-473.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-473.png,Normal/masks/Normal-473.png,True,False,299,299,L,123.91,a5ebd0c594e99e1425eaab175fa2c142
Normal-4730.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4730.png,Normal/images/Normal-4730.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4730.png,Normal/masks/Normal-4730.png,True,False,299,299,L,107.67,01e7a6b7ae43d52882c6b68f69e7a4ae
Normal-4731.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4731.png,Normal/images/Normal-4731.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4731.png,Normal/masks/Normal-4731.png,True,False,299,299,L,128.46,8bb16d3e84d084b4a40dc007b400765e
Normal-4732.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4732.png,Normal/images/Normal-4732.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4732.png,Normal/masks/Normal-4732.png,True,False,299,299,L,119.76,b6442ac2060231140910a984608f8a26
Normal-4733.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4733.png,Normal/images/Normal-4733.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4733.png,Normal/masks/Normal-4733.png,True,False,299,299,L,138.94,c60ddf73b48efa866f406e2bdd743dc1
Normal-4734.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4734.png,Normal/images/Normal-4734.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4734.png,Normal/masks/Normal-4734.png,True,False,299,299,L,113.3,916bfb97ce1b5b327d0b8d0e7d10ba4a
Normal-4735.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4735.png,Normal/images/Normal-4735.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4735.png,Normal/masks/Normal-4735.png,True,False,299,299,L,115.58,a822abb800a34e0b95db0f5f1dd2f4d0
Normal-4736.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4736.png,Normal/images/Normal-4736.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4736.png,Normal/masks/Normal-4736.png,True,False,299,299,L,115.1,0896c4f1db790198322832909b31f03e
Normal-4737.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4737.png,Normal/images/Normal-4737.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4737.png,Normal/masks/Normal-4737.png,True,False,299,299,L,150.44,921aae1627840a18ddad14b3d19b8e24
Normal-4738.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4738.png,Normal/images/Normal-4738.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4738.png,Normal/masks/Normal-4738.png,True,False,299,299,L,106.6,e465d754874ba0eebbfe169e49eafb40
Normal-4739.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4739.png,Normal/images/Normal-4739.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4739.png,Normal/masks/Normal-4739.png,True,False,299,299,L,174.04,d4da1ee7a82204d979a0c93c6437f077
Normal-474.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-474.png,Normal/images/Normal-474.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-474.png,Normal/masks/Normal-474.png,True,False,299,299,L,121.69,d884fd732c0458061b77a4ffb3d07a78
Normal-4740.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4740.png,Normal/images/Normal-4740.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4740.png,Normal/masks/Normal-4740.png,True,False,299,299,L,160.91,3ad97d45c6d7e46faaeeae1c426fe88e
Normal-4741.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4741.png,Normal/images/Normal-4741.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4741.png,Normal/masks/Normal-4741.png,True,False,299,299,L,125.66,cc9ce5607db64262bfd2b489e92f98aa
Normal-4742.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4742.png,Normal/images/Normal-4742.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4742.png,Normal/masks/Normal-4742.png,True,False,299,299,L,120.56,beb0a286b6459815ac3964301f8d2fdd
Normal-4743.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4743.png,Normal/images/Normal-4743.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4743.png,Normal/masks/Normal-4743.png,True,False,299,299,L,109.42,4daf85558c9321b9f2503d6c6cf5f895
Normal-4744.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4744.png,Normal/images/Normal-4744.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4744.png,Normal/masks/Normal-4744.png,True,False,299,299,L,170.25,71ad639a9c89a925913ba9f4187f9b97
Normal-4745.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4745.png,Normal/images/Normal-4745.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4745.png,Normal/masks/Normal-4745.png,True,False,299,299,L,86.5,0b45a3daf2c69e597cbf5df8b3a3b7f6
Normal-4746.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4746.png,Normal/images/Normal-4746.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4746.png,Normal/masks/Normal-4746.png,True,False,299,299,L,130.01,977641e7dbf6edd6cb9e229b0e58cf94
Normal-4747.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4747.png,Normal/images/Normal-4747.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4747.png,Normal/masks/Normal-4747.png,True,False,299,299,L,115.56,1ebb11309fa29e19ce19fb42de8a8175
Normal-4748.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4748.png,Normal/images/Normal-4748.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4748.png,Normal/masks/Normal-4748.png,True,False,299,299,L,128.86,b996e79f525f816cfcdc3c17bd95b0a0
Normal-4749.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4749.png,Normal/images/Normal-4749.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4749.png,Normal/masks/Normal-4749.png,True,False,299,299,L,123.72,dde3ef0db4278b4852a04191e2eeee63
Normal-475.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-475.png,Normal/images/Normal-475.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-475.png,Normal/masks/Normal-475.png,True,False,299,299,L,107.23,c82eaef829bb5c214a5c2a796c48c1b3
Normal-4750.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4750.png,Normal/images/Normal-4750.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4750.png,Normal/masks/Normal-4750.png,True,False,299,299,L,140.73,4b458f80575f3f99b35cbf80afd1a474
Normal-4751.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4751.png,Normal/images/Normal-4751.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4751.png,Normal/masks/Normal-4751.png,True,False,299,299,L,110.64,cb17212e33635d9b81843b1d9482276b
Normal-4752.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4752.png,Normal/images/Normal-4752.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4752.png,Normal/masks/Normal-4752.png,True,False,299,299,L,169.61,ba5a70fa80e4e027e590c5b5c5a9b4d2
Normal-4753.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4753.png,Normal/images/Normal-4753.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4753.png,Normal/masks/Normal-4753.png,True,False,299,299,L,120.34,f062c8875b1b2c1d05f5c0d4e5c92180
Normal-4754.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4754.png,Normal/images/Normal-4754.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4754.png,Normal/masks/Normal-4754.png,True,False,299,299,L,114.22,df86334a1373b4d8ab669e75c32bdbb2
Normal-4755.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4755.png,Normal/images/Normal-4755.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4755.png,Normal/masks/Normal-4755.png,True,False,299,299,L,179.37,669ef3b3517590ac1bdb48d3b4cacba7
Normal-4756.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4756.png,Normal/images/Normal-4756.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4756.png,Normal/masks/Normal-4756.png,True,False,299,299,L,120.56,d4c82d0054de70987cc10b43f3c1d347
Normal-4757.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4757.png,Normal/images/Normal-4757.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4757.png,Normal/masks/Normal-4757.png,True,False,299,299,L,153.24,e98804a0d6e558347af0351b620d0c3d
Normal-4758.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4758.png,Normal/images/Normal-4758.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4758.png,Normal/masks/Normal-4758.png,True,False,299,299,L,138.96,19276f49bd342390274565675b4a59ff
Normal-4759.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4759.png,Normal/images/Normal-4759.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4759.png,Normal/masks/Normal-4759.png,True,False,299,299,L,188.31,b0af82dc605afbce77177d3dd15541a2
Normal-476.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-476.png,Normal/images/Normal-476.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-476.png,Normal/masks/Normal-476.png,True,False,299,299,L,121.46,5442fdb66e28d9a2caaa641884bfda67
Normal-4760.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4760.png,Normal/images/Normal-4760.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4760.png,Normal/masks/Normal-4760.png,True,False,299,299,L,191.47,24739f7db4151c02d8cd3b4f581ae1d4
Normal-4761.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4761.png,Normal/images/Normal-4761.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4761.png,Normal/masks/Normal-4761.png,True,False,299,299,L,107.39,ff879d669ec58bdaac2539107217ae60
Normal-4762.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4762.png,Normal/images/Normal-4762.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4762.png,Normal/masks/Normal-4762.png,True,False,299,299,L,139.9,ebdff91bcca96b222ddc09a0be9ca85a
Normal-4763.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4763.png,Normal/images/Normal-4763.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4763.png,Normal/masks/Normal-4763.png,True,False,299,299,L,178.13,9f2ffa51454f3200b9b128798a3af7de
Normal-4764.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4764.png,Normal/images/Normal-4764.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4764.png,Normal/masks/Normal-4764.png,True,False,299,299,L,152.9,5ef45870989a6eba78184b34adadb45a
Normal-4765.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4765.png,Normal/images/Normal-4765.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4765.png,Normal/masks/Normal-4765.png,True,False,299,299,L,105.52,cc1d17cb11ef514027ba7fc16d0964d8
Normal-4766.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4766.png,Normal/images/Normal-4766.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4766.png,Normal/masks/Normal-4766.png,True,False,299,299,L,123.18,8fa9a88950220cfa0031792ae7ef4256
Normal-4767.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4767.png,Normal/images/Normal-4767.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4767.png,Normal/masks/Normal-4767.png,True,False,299,299,L,148.95,34e9c6039ae1ba57e3792ace352aedf5
Normal-4768.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4768.png,Normal/images/Normal-4768.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4768.png,Normal/masks/Normal-4768.png,True,False,299,299,L,114.04,ab4f43b44acf82d90708fe273aafa27d
Normal-4769.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4769.png,Normal/images/Normal-4769.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4769.png,Normal/masks/Normal-4769.png,True,False,299,299,L,132.74,1505a76e2df536c27c5bb727b1813ae4
Normal-477.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-477.png,Normal/images/Normal-477.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-477.png,Normal/masks/Normal-477.png,True,False,299,299,L,115.95,5c64d09e774ee3953183035bf3a22e9f
Normal-4770.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4770.png,Normal/images/Normal-4770.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4770.png,Normal/masks/Normal-4770.png,True,False,299,299,L,119.7,1da97fac93aa505af61cfd141a785e7a
Normal-4771.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4771.png,Normal/images/Normal-4771.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4771.png,Normal/masks/Normal-4771.png,True,False,299,299,L,118.25,6d641d0d109e43595bbf9f40723e551f
Normal-4772.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4772.png,Normal/images/Normal-4772.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4772.png,Normal/masks/Normal-4772.png,True,False,299,299,L,168.12,e471b7ae6f46413de69920bcca70c16f
Normal-4773.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4773.png,Normal/images/Normal-4773.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4773.png,Normal/masks/Normal-4773.png,True,False,299,299,L,132.04,25518f85ad2c27f39e70fed42ad51afa
Normal-4774.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4774.png,Normal/images/Normal-4774.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4774.png,Normal/masks/Normal-4774.png,True,False,299,299,L,154.99,af5f63707dec91df6a3180a2e0827ecd
Normal-4775.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4775.png,Normal/images/Normal-4775.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4775.png,Normal/masks/Normal-4775.png,True,False,299,299,L,110.79,eb3d0fedbe8a9482af8599ad4617683a
Normal-4776.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4776.png,Normal/images/Normal-4776.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4776.png,Normal/masks/Normal-4776.png,True,False,299,299,L,134.67,ce87dc0f28536c0993ab5d42f34e3d1a
Normal-4777.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4777.png,Normal/images/Normal-4777.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4777.png,Normal/masks/Normal-4777.png,True,False,299,299,L,119.06,3d008d3990a618304f1c66c152cf5aed
Normal-4778.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4778.png,Normal/images/Normal-4778.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4778.png,Normal/masks/Normal-4778.png,True,False,299,299,L,95.77,31bdccbc6227d8c2c75b9df08f8ebcde
Normal-4779.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4779.png,Normal/images/Normal-4779.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4779.png,Normal/masks/Normal-4779.png,True,False,299,299,L,119.03,4f77e19044a2d8a234ff07b42611421d
Normal-478.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-478.png,Normal/images/Normal-478.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-478.png,Normal/masks/Normal-478.png,True,False,299,299,L,109.47,8af000d53df643712ea96bde10e20302
Normal-4780.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4780.png,Normal/images/Normal-4780.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4780.png,Normal/masks/Normal-4780.png,True,False,299,299,L,103.01,77be8baaa451baeb5f13950b5a0950be
Normal-4781.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4781.png,Normal/images/Normal-4781.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4781.png,Normal/masks/Normal-4781.png,True,False,299,299,L,103.94,a516bebdd000522d1e831769268f2b38
Normal-4782.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4782.png,Normal/images/Normal-4782.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4782.png,Normal/masks/Normal-4782.png,True,False,299,299,L,123.08,b127b17e0e79c3568addc24efdce75ae
Normal-4783.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4783.png,Normal/images/Normal-4783.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4783.png,Normal/masks/Normal-4783.png,True,False,299,299,L,118.94,57a34dbaba6de104a9c5a96147a6cd02
Normal-4784.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4784.png,Normal/images/Normal-4784.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4784.png,Normal/masks/Normal-4784.png,True,False,299,299,L,130.32,15790401f01ac6847b43f61e3cfa5209
Normal-4785.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4785.png,Normal/images/Normal-4785.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4785.png,Normal/masks/Normal-4785.png,True,False,299,299,L,168.83,4612fc2f5052c9bc065560d07f3ff4e4
Normal-4786.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4786.png,Normal/images/Normal-4786.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4786.png,Normal/masks/Normal-4786.png,True,False,299,299,L,120.28,10080280b9e81d14a1971876f9e1c6bc
Normal-4787.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4787.png,Normal/images/Normal-4787.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4787.png,Normal/masks/Normal-4787.png,True,False,299,299,L,115.23,10fe4d0757b6a57d677d9f606ff8261b
Normal-4788.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4788.png,Normal/images/Normal-4788.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4788.png,Normal/masks/Normal-4788.png,True,False,299,299,L,130.02,4dd57732c4a246de75d9d2f980f1ecf8
Normal-4789.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4789.png,Normal/images/Normal-4789.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4789.png,Normal/masks/Normal-4789.png,True,False,299,299,L,172.96,03c37d08b05652fd1c4d706dd469e350
Normal-479.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-479.png,Normal/images/Normal-479.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-479.png,Normal/masks/Normal-479.png,True,False,299,299,L,118.81,52674daecac12e38d310636c22f39e6a
Normal-4790.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4790.png,Normal/images/Normal-4790.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4790.png,Normal/masks/Normal-4790.png,True,False,299,299,L,119.48,4ec1b0983115e13c9bce96cd26a87455
Normal-4791.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4791.png,Normal/images/Normal-4791.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4791.png,Normal/masks/Normal-4791.png,True,False,299,299,L,144.16,52ae90395f9cf3b76f4ecf41a7a387b0
Normal-4792.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4792.png,Normal/images/Normal-4792.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4792.png,Normal/masks/Normal-4792.png,True,False,299,299,L,113.92,978d8dfe8fdfa7d06a9ff8d4fd2fa2a6
Normal-4793.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4793.png,Normal/images/Normal-4793.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4793.png,Normal/masks/Normal-4793.png,True,False,299,299,L,159.25,dbd565ab9153dc466b04bdcec7fe96cf
Normal-4794.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4794.png,Normal/images/Normal-4794.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4794.png,Normal/masks/Normal-4794.png,True,False,299,299,L,144.12,7defd7f99b28d3bb15065a07300c41b7
Normal-4795.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4795.png,Normal/images/Normal-4795.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4795.png,Normal/masks/Normal-4795.png,True,False,299,299,L,84.43,9ef07663b93088473a1d707063d3b8e7
Normal-4796.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4796.png,Normal/images/Normal-4796.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4796.png,Normal/masks/Normal-4796.png,True,False,299,299,L,139.69,79df13f3ceed01b29a8b1697f864d653
Normal-4797.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4797.png,Normal/images/Normal-4797.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4797.png,Normal/masks/Normal-4797.png,True,False,299,299,L,128.65,62a2a193269ad3f03180fcba4f87ed04
Normal-4798.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4798.png,Normal/images/Normal-4798.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4798.png,Normal/masks/Normal-4798.png,True,False,299,299,L,168.05,14547ee80c785d2a4dfbf78c1bb666cb
Normal-4799.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4799.png,Normal/images/Normal-4799.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4799.png,Normal/masks/Normal-4799.png,True,False,299,299,L,120.02,1cdd8a0fc4701896ad4edcef1a98073f
Normal-48.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-48.png,Normal/images/Normal-48.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-48.png,Normal/masks/Normal-48.png,True,False,299,299,L,110.32,4d86586078e8ccb7ff9fefb6b8d3aec9
Normal-480.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-480.png,Normal/images/Normal-480.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-480.png,Normal/masks/Normal-480.png,True,False,299,299,L,134.97,b98beb1728b61f609cd2ce55ea7cbc15
Normal-4800.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4800.png,Normal/images/Normal-4800.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4800.png,Normal/masks/Normal-4800.png,True,False,299,299,L,158.07,11988106b3d1695a53fb1297e04ba4a3
Normal-4801.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4801.png,Normal/images/Normal-4801.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4801.png,Normal/masks/Normal-4801.png,True,False,299,299,L,138.95,302084e8dda6e5cdf780f1ba78333a31
Normal-4802.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4802.png,Normal/images/Normal-4802.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4802.png,Normal/masks/Normal-4802.png,True,False,299,299,L,135.98,dc0c19d38d26c83ca5a2cc56ba0ec29e
Normal-4803.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4803.png,Normal/images/Normal-4803.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4803.png,Normal/masks/Normal-4803.png,True,False,299,299,L,80.43,1a35f390029fdc6d8951620a88b051b1
Normal-4804.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4804.png,Normal/images/Normal-4804.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4804.png,Normal/masks/Normal-4804.png,True,False,299,299,L,121.54,9aed10371c0eb77add6f84df411abacd
Normal-4805.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4805.png,Normal/images/Normal-4805.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4805.png,Normal/masks/Normal-4805.png,True,False,299,299,L,169.07,2e3e137f4bc010562784697f3042265f
Normal-4806.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4806.png,Normal/images/Normal-4806.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4806.png,Normal/masks/Normal-4806.png,True,False,299,299,L,96.3,d384f545cd0ce863a275b446b0407e99
Normal-4807.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4807.png,Normal/images/Normal-4807.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4807.png,Normal/masks/Normal-4807.png,True,False,299,299,L,126.1,42737e4eac93da481197ba83752995f3
Normal-4808.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4808.png,Normal/images/Normal-4808.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4808.png,Normal/masks/Normal-4808.png,True,False,299,299,L,107.84,0abb92856323a0d8125fc911a0a28d4b
Normal-4809.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4809.png,Normal/images/Normal-4809.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4809.png,Normal/masks/Normal-4809.png,True,False,299,299,L,158.86,33bf2d18e484b909a83b0c669d810de9
Normal-481.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-481.png,Normal/images/Normal-481.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-481.png,Normal/masks/Normal-481.png,True,False,299,299,L,130.52,35cb946f5d942caf5d1c6d9de7a9f7dd
Normal-4810.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4810.png,Normal/images/Normal-4810.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4810.png,Normal/masks/Normal-4810.png,True,False,299,299,L,98.1,8a1c73f8515680329ad6ccf91567980d
Normal-4811.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4811.png,Normal/images/Normal-4811.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4811.png,Normal/masks/Normal-4811.png,True,False,299,299,L,120.1,a1babf6d79520c92fad8bad0fee23fe8
Normal-4812.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4812.png,Normal/images/Normal-4812.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4812.png,Normal/masks/Normal-4812.png,True,False,299,299,L,127.71,af13539bea058ddcb551421573a96876
Normal-4813.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4813.png,Normal/images/Normal-4813.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4813.png,Normal/masks/Normal-4813.png,True,False,299,299,L,127.55,ca5cc3ee8cb95606df9e1aedb7a0a1b4
Normal-4814.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4814.png,Normal/images/Normal-4814.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4814.png,Normal/masks/Normal-4814.png,True,False,299,299,L,151.65,200faadbc71c9205c06c0a37c17577bd
Normal-4815.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4815.png,Normal/images/Normal-4815.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4815.png,Normal/masks/Normal-4815.png,True,False,299,299,L,99.39,5cd6d458d3092a279e7aceee0782926a
Normal-4816.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4816.png,Normal/images/Normal-4816.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4816.png,Normal/masks/Normal-4816.png,True,False,299,299,L,106.36,e58b296851baad9cea5739011c456556
Normal-4817.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4817.png,Normal/images/Normal-4817.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4817.png,Normal/masks/Normal-4817.png,True,False,299,299,L,125.54,2da3d6fe1bea8e873f4093e876562c09
Normal-4818.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4818.png,Normal/images/Normal-4818.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4818.png,Normal/masks/Normal-4818.png,True,False,299,299,L,125.8,99f496eb7ecca8dd80d9b354a2454d1d
Normal-4819.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4819.png,Normal/images/Normal-4819.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4819.png,Normal/masks/Normal-4819.png,True,False,299,299,L,156.69,f168a4235a7b661334ae45d57ff7a45e
Normal-482.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-482.png,Normal/images/Normal-482.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-482.png,Normal/masks/Normal-482.png,True,False,299,299,L,131.15,6f42df080baa145fddc2b11bab0caee7
Normal-4820.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4820.png,Normal/images/Normal-4820.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4820.png,Normal/masks/Normal-4820.png,True,False,299,299,L,130.25,a5c88f21dfc55e5883d3c5c03fa021e9
Normal-4821.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4821.png,Normal/images/Normal-4821.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4821.png,Normal/masks/Normal-4821.png,True,False,299,299,L,125.81,6b134f1c1907c50917c70935e2862a96
Normal-4822.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4822.png,Normal/images/Normal-4822.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4822.png,Normal/masks/Normal-4822.png,True,False,299,299,L,161.14,36a1227052a67f4ae014a3f1630100fc
Normal-4823.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4823.png,Normal/images/Normal-4823.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4823.png,Normal/masks/Normal-4823.png,True,False,299,299,L,130.19,76daf762569e123b7fe360a43d1e726e
Normal-4824.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4824.png,Normal/images/Normal-4824.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4824.png,Normal/masks/Normal-4824.png,True,False,299,299,L,118.37,c42bc818930800e4801d7b585c53e425
Normal-4825.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4825.png,Normal/images/Normal-4825.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4825.png,Normal/masks/Normal-4825.png,True,False,299,299,L,119.83,0304d557b06c98d7847286fdec4f111f
Normal-4826.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4826.png,Normal/images/Normal-4826.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4826.png,Normal/masks/Normal-4826.png,True,False,299,299,L,113.93,be9f1714148ae9af86b4559febab1aba
Normal-4827.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4827.png,Normal/images/Normal-4827.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4827.png,Normal/masks/Normal-4827.png,True,False,299,299,L,181.68,9bf31565c28a67e17e40eafb74f95375
Normal-4828.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4828.png,Normal/images/Normal-4828.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4828.png,Normal/masks/Normal-4828.png,True,False,299,299,L,112.69,b5f39350e1d93f23be4371155b789249
Normal-4829.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4829.png,Normal/images/Normal-4829.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4829.png,Normal/masks/Normal-4829.png,True,False,299,299,L,120.98,eb766ecd76d77284f95d83e700faf2e9
Normal-483.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-483.png,Normal/images/Normal-483.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-483.png,Normal/masks/Normal-483.png,True,False,299,299,L,155.85,e9a1707b1f47f9b5597fff0a8be9503c
Normal-4830.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4830.png,Normal/images/Normal-4830.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4830.png,Normal/masks/Normal-4830.png,True,False,299,299,L,94.1,0369dfd69eabee1c604b1ebe68390ebb
Normal-4831.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4831.png,Normal/images/Normal-4831.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4831.png,Normal/masks/Normal-4831.png,True,False,299,299,L,99.06,c2c18935de1242452335eb544719e6e3
Normal-4832.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4832.png,Normal/images/Normal-4832.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4832.png,Normal/masks/Normal-4832.png,True,False,299,299,L,130.35,631b6d3c59d5d3fb6c2077eb143969ea
Normal-4833.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4833.png,Normal/images/Normal-4833.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4833.png,Normal/masks/Normal-4833.png,True,False,299,299,L,96.15,c17eacff384af9643541fccda4564012
Normal-4834.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4834.png,Normal/images/Normal-4834.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4834.png,Normal/masks/Normal-4834.png,True,False,299,299,L,118.68,3bd6e155eeab646f29e2e5332979c0a1
Normal-4835.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4835.png,Normal/images/Normal-4835.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4835.png,Normal/masks/Normal-4835.png,True,False,299,299,L,116.93,3ac0d04b01266c6df3d28a1d22c2333c
Normal-4836.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4836.png,Normal/images/Normal-4836.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4836.png,Normal/masks/Normal-4836.png,True,False,299,299,L,124.28,43441fec418bb1451039d76350920f62
Normal-4837.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4837.png,Normal/images/Normal-4837.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4837.png,Normal/masks/Normal-4837.png,True,False,299,299,L,126.51,743544589b6bd21d5d78cdbc0f3498f0
Normal-4838.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4838.png,Normal/images/Normal-4838.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4838.png,Normal/masks/Normal-4838.png,True,False,299,299,L,121.95,6a9f3ac5b7b7895eb68e9e4dfe224b5e
Normal-4839.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4839.png,Normal/images/Normal-4839.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4839.png,Normal/masks/Normal-4839.png,True,False,299,299,L,116.38,93760eee8ac77d77621e0a2af42c047b
Normal-484.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-484.png,Normal/images/Normal-484.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-484.png,Normal/masks/Normal-484.png,True,False,299,299,L,117.15,0d7751c9d246c73019574b9b8fcc46c7
Normal-4840.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4840.png,Normal/images/Normal-4840.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4840.png,Normal/masks/Normal-4840.png,True,False,299,299,L,169.65,ec30ac84fcad691c2b2fc7a5bb75ad5a
Normal-4841.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4841.png,Normal/images/Normal-4841.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4841.png,Normal/masks/Normal-4841.png,True,False,299,299,L,138.29,0ac0a09099f00fd3cd2c390a80edaa68
Normal-4842.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4842.png,Normal/images/Normal-4842.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4842.png,Normal/masks/Normal-4842.png,True,False,299,299,L,121.36,64e1d526da4a9a2ed84845ae0c6aef60
Normal-4843.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4843.png,Normal/images/Normal-4843.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4843.png,Normal/masks/Normal-4843.png,True,False,299,299,L,181.55,bb017dfc13998e6fb17a08097dae0a06
Normal-4844.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4844.png,Normal/images/Normal-4844.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4844.png,Normal/masks/Normal-4844.png,True,False,299,299,L,120.98,cd2848b8d36c54e1c18b5c07536d7fcb
Normal-4845.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4845.png,Normal/images/Normal-4845.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4845.png,Normal/masks/Normal-4845.png,True,False,299,299,L,146.98,e0697714b4f7fffe713f2778227db3eb
Normal-4846.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4846.png,Normal/images/Normal-4846.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4846.png,Normal/masks/Normal-4846.png,True,False,299,299,L,121.62,272b799d701402f97f18b138cf9abd60
Normal-4847.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4847.png,Normal/images/Normal-4847.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4847.png,Normal/masks/Normal-4847.png,True,False,299,299,L,125.42,4e4d8feec066c6dd2534f1946f6b797c
Normal-4848.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4848.png,Normal/images/Normal-4848.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4848.png,Normal/masks/Normal-4848.png,True,False,299,299,L,66.74,a7806fe7f7f3511062f4a7cd17e9b577
Normal-4849.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4849.png,Normal/images/Normal-4849.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4849.png,Normal/masks/Normal-4849.png,True,False,299,299,L,134.8,da6fcc47b08d8e62f7fc3007e3aaee47
Normal-485.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-485.png,Normal/images/Normal-485.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-485.png,Normal/masks/Normal-485.png,True,False,299,299,L,110.81,62fe50730ca38b1680c3638a90feff27
Normal-4850.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4850.png,Normal/images/Normal-4850.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4850.png,Normal/masks/Normal-4850.png,True,False,299,299,L,149.51,2159c6da493d0af96caf1e8599e97a4d
Normal-4851.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4851.png,Normal/images/Normal-4851.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4851.png,Normal/masks/Normal-4851.png,True,False,299,299,L,177.36,ec4dd358a32690588229619eded4d0b3
Normal-4852.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4852.png,Normal/images/Normal-4852.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4852.png,Normal/masks/Normal-4852.png,True,False,299,299,L,124.42,0904f6ef56b654c4a2e55b594beffad9
Normal-4853.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4853.png,Normal/images/Normal-4853.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4853.png,Normal/masks/Normal-4853.png,True,False,299,299,L,135.28,6064dc786bf884ea2a9d09a621de75f1
Normal-4854.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4854.png,Normal/images/Normal-4854.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4854.png,Normal/masks/Normal-4854.png,True,False,299,299,L,114.72,cd3bbb62ef3041d54449f4935e6e830e
Normal-4855.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4855.png,Normal/images/Normal-4855.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4855.png,Normal/masks/Normal-4855.png,True,False,299,299,L,117.33,0340e5e4997c47731463000266d76eee
Normal-4856.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4856.png,Normal/images/Normal-4856.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4856.png,Normal/masks/Normal-4856.png,True,False,299,299,L,130.47,55fb6712def8cbae996698d5a4e6e7cb
Normal-4857.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4857.png,Normal/images/Normal-4857.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4857.png,Normal/masks/Normal-4857.png,True,False,299,299,L,134.57,4513c90a762b9f4c0dac38b205c7444e
Normal-4858.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4858.png,Normal/images/Normal-4858.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4858.png,Normal/masks/Normal-4858.png,True,False,299,299,L,136.21,4676a77752ac28c089b38d73cfe57c6b
Normal-4859.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4859.png,Normal/images/Normal-4859.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4859.png,Normal/masks/Normal-4859.png,True,False,299,299,L,94.15,aaf7a7505724f9c608b385ca97251a88
Normal-486.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-486.png,Normal/images/Normal-486.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-486.png,Normal/masks/Normal-486.png,True,False,299,299,L,108.45,44a9ff06569e56016a8df456fe5afd91
Normal-4860.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4860.png,Normal/images/Normal-4860.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4860.png,Normal/masks/Normal-4860.png,True,False,299,299,L,103.85,3468c118fd882a7a93ded420c9e8515e
Normal-4861.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4861.png,Normal/images/Normal-4861.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4861.png,Normal/masks/Normal-4861.png,True,False,299,299,L,150.63,4f371d65b77c3ef17abcbd124e5ff39f
Normal-4862.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4862.png,Normal/images/Normal-4862.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4862.png,Normal/masks/Normal-4862.png,True,False,299,299,L,165.02,425af325f805eae36707c71544f4a4f3
Normal-4863.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4863.png,Normal/images/Normal-4863.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4863.png,Normal/masks/Normal-4863.png,True,False,299,299,L,125.93,b6b478021aff4cf1c20c0cfc384d6ab1
Normal-4864.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4864.png,Normal/images/Normal-4864.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4864.png,Normal/masks/Normal-4864.png,True,False,299,299,L,120.53,7b1716c22963c7b8892bebc83b2540f5
Normal-4865.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4865.png,Normal/images/Normal-4865.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4865.png,Normal/masks/Normal-4865.png,True,False,299,299,L,122.12,a9776f79785f728b8ec2942e4cac3d36
Normal-4866.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4866.png,Normal/images/Normal-4866.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4866.png,Normal/masks/Normal-4866.png,True,False,299,299,L,142.74,c6c587420185a6a355dd512af918b294
Normal-4867.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4867.png,Normal/images/Normal-4867.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4867.png,Normal/masks/Normal-4867.png,True,False,299,299,L,129.79,d505bb70f349797d25da3f8c8d1b508d
Normal-4868.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4868.png,Normal/images/Normal-4868.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4868.png,Normal/masks/Normal-4868.png,True,False,299,299,L,105.03,d1d335fdfb23812adb2b5e13cec9f7dd
Normal-4869.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4869.png,Normal/images/Normal-4869.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4869.png,Normal/masks/Normal-4869.png,True,False,299,299,L,143.59,a9d2453e598be99ed77e16af9ca557d7
Normal-487.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-487.png,Normal/images/Normal-487.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-487.png,Normal/masks/Normal-487.png,True,False,299,299,L,121.69,30ec789661886a6fd3a4d25f29903eb9
Normal-4870.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4870.png,Normal/images/Normal-4870.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4870.png,Normal/masks/Normal-4870.png,True,False,299,299,L,127.35,ce676b771dfd696a779641ab14636cde
Normal-4871.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4871.png,Normal/images/Normal-4871.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4871.png,Normal/masks/Normal-4871.png,True,False,299,299,L,165.89,6d51bb9eee98dd549cf0460e4f97ae21
Normal-4872.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4872.png,Normal/images/Normal-4872.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4872.png,Normal/masks/Normal-4872.png,True,False,299,299,L,130.8,25d73485b7e3d2f5a567bc27055a7021
Normal-4873.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4873.png,Normal/images/Normal-4873.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4873.png,Normal/masks/Normal-4873.png,True,False,299,299,L,149.57,799586bd1dc5f609db85d6a66402ab46
Normal-4874.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4874.png,Normal/images/Normal-4874.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4874.png,Normal/masks/Normal-4874.png,True,False,299,299,L,172.05,562796d149727b1dbf25c34b4a208083
Normal-4875.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4875.png,Normal/images/Normal-4875.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4875.png,Normal/masks/Normal-4875.png,True,False,299,299,L,163.39,608cca01d16cd1d09554c41eea61415b
Normal-4876.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4876.png,Normal/images/Normal-4876.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4876.png,Normal/masks/Normal-4876.png,True,False,299,299,L,113.33,0455b563a5629fee3571a1b5b882d5ef
Normal-4877.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4877.png,Normal/images/Normal-4877.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4877.png,Normal/masks/Normal-4877.png,True,False,299,299,L,163.99,be14a92a2371054ab10c01d2da3df4d2
Normal-4878.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4878.png,Normal/images/Normal-4878.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4878.png,Normal/masks/Normal-4878.png,True,False,299,299,L,118.28,c38e23693947ac6f36b91d5920adee7e
Normal-4879.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4879.png,Normal/images/Normal-4879.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4879.png,Normal/masks/Normal-4879.png,True,False,299,299,L,172.89,24a1780f9d4732160a7bc5c369cd62d7
Normal-488.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-488.png,Normal/images/Normal-488.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-488.png,Normal/masks/Normal-488.png,True,False,299,299,L,134.2,6cd74f3fe26aaaaa46419d5e129a88a9
Normal-4880.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4880.png,Normal/images/Normal-4880.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4880.png,Normal/masks/Normal-4880.png,True,False,299,299,L,127.05,95a923fccf1e465341d97966bbd4c4e3
Normal-4881.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4881.png,Normal/images/Normal-4881.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4881.png,Normal/masks/Normal-4881.png,True,False,299,299,L,135.28,bc5d9172de785493712787a3fe18e840
Normal-4882.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4882.png,Normal/images/Normal-4882.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4882.png,Normal/masks/Normal-4882.png,True,False,299,299,L,173.43,9fbf85487ec03d6faf5c855ba1f67a41
Normal-4883.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4883.png,Normal/images/Normal-4883.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4883.png,Normal/masks/Normal-4883.png,True,False,299,299,L,73.56,416cfaaa8dd6e9bcb2cc3a67e1da9ad8
Normal-4884.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4884.png,Normal/images/Normal-4884.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4884.png,Normal/masks/Normal-4884.png,True,False,299,299,L,106.64,e996a8e2b50986c847ed52ba4e7d45b3
Normal-4885.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4885.png,Normal/images/Normal-4885.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4885.png,Normal/masks/Normal-4885.png,True,False,299,299,L,151.67,478cf9030acba5d811ea615288f3ba89
Normal-4886.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4886.png,Normal/images/Normal-4886.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4886.png,Normal/masks/Normal-4886.png,True,False,299,299,L,120.23,acc350aca6b290c0726ee736fa918a24
Normal-4887.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4887.png,Normal/images/Normal-4887.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4887.png,Normal/masks/Normal-4887.png,True,False,299,299,L,121.32,c9f7912014cc441643175eb4a33996e7
Normal-4888.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4888.png,Normal/images/Normal-4888.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4888.png,Normal/masks/Normal-4888.png,True,False,299,299,L,155.78,6a1c2b89c1481dfee9950e2f9511d17a
Normal-4889.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4889.png,Normal/images/Normal-4889.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4889.png,Normal/masks/Normal-4889.png,True,False,299,299,L,141.66,4321c0cc8d9d78346c7eb6acf4456a9f
Normal-489.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-489.png,Normal/images/Normal-489.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-489.png,Normal/masks/Normal-489.png,True,False,299,299,L,125.87,872076bfd562e9a07cc6bec1a0a34f14
Normal-4890.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4890.png,Normal/images/Normal-4890.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4890.png,Normal/masks/Normal-4890.png,True,False,299,299,L,172.1,38e0c1569be4546380f52dbba571b583
Normal-4891.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4891.png,Normal/images/Normal-4891.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4891.png,Normal/masks/Normal-4891.png,True,False,299,299,L,104.32,6d13728d21945a59712db882484d0d0a
Normal-4892.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4892.png,Normal/images/Normal-4892.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4892.png,Normal/masks/Normal-4892.png,True,False,299,299,L,152.73,8cf89922e8b49eef97b501707518cbaa
Normal-4893.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4893.png,Normal/images/Normal-4893.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4893.png,Normal/masks/Normal-4893.png,True,False,299,299,L,117.48,e4805578227513602cd95dc0e1334da4
Normal-4894.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4894.png,Normal/images/Normal-4894.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4894.png,Normal/masks/Normal-4894.png,True,False,299,299,L,150.98,2231c9f5a83bcea50fbe1430f880f307
Normal-4895.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4895.png,Normal/images/Normal-4895.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4895.png,Normal/masks/Normal-4895.png,True,False,299,299,L,112.91,208447d39aea56b40e9542c46b9e8468
Normal-4896.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4896.png,Normal/images/Normal-4896.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4896.png,Normal/masks/Normal-4896.png,True,False,299,299,L,109.34,2e6897b641da414872fa4d84481d1824
Normal-4897.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4897.png,Normal/images/Normal-4897.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4897.png,Normal/masks/Normal-4897.png,True,False,299,299,L,117.1,de60892d8261954e3e6a795f64b44413
Normal-4898.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4898.png,Normal/images/Normal-4898.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4898.png,Normal/masks/Normal-4898.png,True,False,299,299,L,151.69,9b74bb6e3b251e08f8c29edefe66c855
Normal-4899.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4899.png,Normal/images/Normal-4899.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4899.png,Normal/masks/Normal-4899.png,True,False,299,299,L,122.09,5f1524a0d6e3cb6fce1dd05b489194f2
Normal-49.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-49.png,Normal/images/Normal-49.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-49.png,Normal/masks/Normal-49.png,True,False,299,299,L,128.29,1333c1305303988bea59aaa4a3910173
Normal-490.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-490.png,Normal/images/Normal-490.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-490.png,Normal/masks/Normal-490.png,True,False,299,299,L,138.86,ea30b3efd37787ed9ec33361110eacb7
Normal-4900.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4900.png,Normal/images/Normal-4900.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4900.png,Normal/masks/Normal-4900.png,True,False,299,299,L,122.52,e61f20059dc4ea848ff706b12845559e
Normal-4901.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4901.png,Normal/images/Normal-4901.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4901.png,Normal/masks/Normal-4901.png,True,False,299,299,L,119.55,5ea73a14fc71c9b39e087a5863e14213
Normal-4902.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4902.png,Normal/images/Normal-4902.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4902.png,Normal/masks/Normal-4902.png,True,False,299,299,L,171.42,9578e550420cdde622faa4ce57b01de0
Normal-4903.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4903.png,Normal/images/Normal-4903.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4903.png,Normal/masks/Normal-4903.png,True,False,299,299,L,159.4,315ef1fccd07cac389cf28f24ed56269
Normal-4904.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4904.png,Normal/images/Normal-4904.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4904.png,Normal/masks/Normal-4904.png,True,False,299,299,L,139.34,9725a4f2fc72269e5bb052a23625adf2
Normal-4905.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4905.png,Normal/images/Normal-4905.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4905.png,Normal/masks/Normal-4905.png,True,False,299,299,L,139.95,41777d5f7e6a60c1e21fff53eb2a9a7e
Normal-4906.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4906.png,Normal/images/Normal-4906.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4906.png,Normal/masks/Normal-4906.png,True,False,299,299,L,179.58,73cda54ea737d55e4ff20375eabd1a6b
Normal-4907.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4907.png,Normal/images/Normal-4907.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4907.png,Normal/masks/Normal-4907.png,True,False,299,299,L,143.66,e0557a54ea7823631d914f58e737f38f
Normal-4908.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4908.png,Normal/images/Normal-4908.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4908.png,Normal/masks/Normal-4908.png,True,False,299,299,L,118.21,033e456fb4f8231d0774839b13f6c792
Normal-4909.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4909.png,Normal/images/Normal-4909.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4909.png,Normal/masks/Normal-4909.png,True,False,299,299,L,162.92,df9efcd644c86675e465a9be6292c7be
Normal-491.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-491.png,Normal/images/Normal-491.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-491.png,Normal/masks/Normal-491.png,True,False,299,299,L,123.74,aba4c148bbedcdc5126b176e1c527553
Normal-4910.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4910.png,Normal/images/Normal-4910.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4910.png,Normal/masks/Normal-4910.png,True,False,299,299,L,149.65,5c7bbe1920da46407c9ec88bb6cea44a
Normal-4911.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4911.png,Normal/images/Normal-4911.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4911.png,Normal/masks/Normal-4911.png,True,False,299,299,L,111.51,f82acdd57088923e521d82c526e91922
Normal-4912.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4912.png,Normal/images/Normal-4912.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4912.png,Normal/masks/Normal-4912.png,True,False,299,299,L,123.88,ee52e7df29b31ca112f8c16cdbbfb847
Normal-4913.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4913.png,Normal/images/Normal-4913.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4913.png,Normal/masks/Normal-4913.png,True,False,299,299,L,119.95,3c3b50569ac0aa4600b3fc483c44b3bc
Normal-4914.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4914.png,Normal/images/Normal-4914.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4914.png,Normal/masks/Normal-4914.png,True,False,299,299,L,119.07,58e006e6bc61fd045b39b1a5f78d49e7
Normal-4915.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4915.png,Normal/images/Normal-4915.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4915.png,Normal/masks/Normal-4915.png,True,False,299,299,L,129.49,ebcdc1ad03e75e80cac0d8a54a4f7394
Normal-4916.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4916.png,Normal/images/Normal-4916.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4916.png,Normal/masks/Normal-4916.png,True,False,299,299,L,117.14,5e63620b8d071327d0d8a44c0bfa5b6a
Normal-4917.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4917.png,Normal/images/Normal-4917.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4917.png,Normal/masks/Normal-4917.png,True,False,299,299,L,179.82,f424fc28e33615709f4b68437956fa8f
Normal-4918.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4918.png,Normal/images/Normal-4918.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4918.png,Normal/masks/Normal-4918.png,True,False,299,299,L,107.25,a02b40bc644ca7be157afbb3afdb41ed
Normal-4919.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4919.png,Normal/images/Normal-4919.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4919.png,Normal/masks/Normal-4919.png,True,False,299,299,L,134.74,3c4467faa9507fdddd829b6dd4eeb304
Normal-492.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-492.png,Normal/images/Normal-492.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-492.png,Normal/masks/Normal-492.png,True,False,299,299,L,115.78,362ed9a48be274ab550a7bb1e6292174
Normal-4920.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4920.png,Normal/images/Normal-4920.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4920.png,Normal/masks/Normal-4920.png,True,False,299,299,L,134.11,800e4cc093a6d72e5f074e6b53eb79dd
Normal-4921.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4921.png,Normal/images/Normal-4921.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4921.png,Normal/masks/Normal-4921.png,True,False,299,299,L,126.02,1d0806e46d1f4b7470bb2bf7758c742f
Normal-4922.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4922.png,Normal/images/Normal-4922.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4922.png,Normal/masks/Normal-4922.png,True,False,299,299,L,136.02,fde74d68cb3766518356257a83a08a35
Normal-4923.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4923.png,Normal/images/Normal-4923.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4923.png,Normal/masks/Normal-4923.png,True,False,299,299,L,117.86,4dd930a07c7396995ff5df9f30f88622
Normal-4924.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4924.png,Normal/images/Normal-4924.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4924.png,Normal/masks/Normal-4924.png,True,False,299,299,L,157.59,a41ed44cf6308d8d6913f77bc783f236
Normal-4925.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4925.png,Normal/images/Normal-4925.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4925.png,Normal/masks/Normal-4925.png,True,False,299,299,L,102.03,c1e803bf9f8639ae33ea799185ff0ac4
Normal-4926.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4926.png,Normal/images/Normal-4926.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4926.png,Normal/masks/Normal-4926.png,True,False,299,299,L,138.39,1bdd3c9eb3efa2f651a59cc17bb11397
Normal-4927.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4927.png,Normal/images/Normal-4927.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4927.png,Normal/masks/Normal-4927.png,True,False,299,299,L,116.77,2af3241eab2cd9c4aa77f8d7f42ea5d0
Normal-4928.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4928.png,Normal/images/Normal-4928.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4928.png,Normal/masks/Normal-4928.png,True,False,299,299,L,128.83,b373479848ef8f5be9047843189eca10
Normal-4929.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4929.png,Normal/images/Normal-4929.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4929.png,Normal/masks/Normal-4929.png,True,False,299,299,L,127.23,51155ee549c37dcbf55c99d74cab5128
Normal-493.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-493.png,Normal/images/Normal-493.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-493.png,Normal/masks/Normal-493.png,True,False,299,299,L,126.93,a6a9ff4f78c8efbcd2c03e0bf51da897
Normal-4930.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4930.png,Normal/images/Normal-4930.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4930.png,Normal/masks/Normal-4930.png,True,False,299,299,L,137.1,acf5d775dd6b1036cd9956c0787bd42d
Normal-4931.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4931.png,Normal/images/Normal-4931.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4931.png,Normal/masks/Normal-4931.png,True,False,299,299,L,123.53,0a6ff017f658aabe4927ba92891a4ab3
Normal-4932.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4932.png,Normal/images/Normal-4932.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4932.png,Normal/masks/Normal-4932.png,True,False,299,299,L,93.29,6cc30e1bb35f7c546e81e9d33a61bd4f
Normal-4933.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4933.png,Normal/images/Normal-4933.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4933.png,Normal/masks/Normal-4933.png,True,False,299,299,L,108.58,edf7ecdddc6344a13b837a33b017ff89
Normal-4934.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4934.png,Normal/images/Normal-4934.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4934.png,Normal/masks/Normal-4934.png,True,False,299,299,L,129.72,a84bd30e466d4c09a9c84e754d3558d2
Normal-4935.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4935.png,Normal/images/Normal-4935.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4935.png,Normal/masks/Normal-4935.png,True,False,299,299,L,121.44,d91c18daae73b5a7ac5a41341b20a4b4
Normal-4936.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4936.png,Normal/images/Normal-4936.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4936.png,Normal/masks/Normal-4936.png,True,False,299,299,L,167.28,b4bc120e33ec2985da70efef6b729c26
Normal-4937.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4937.png,Normal/images/Normal-4937.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4937.png,Normal/masks/Normal-4937.png,True,False,299,299,L,167.75,bf66fae59b10177f975a13d4ff1fe5d8
Normal-4938.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4938.png,Normal/images/Normal-4938.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4938.png,Normal/masks/Normal-4938.png,True,False,299,299,L,125.16,cc94b6815411b01df5a30d06606bbe36
Normal-4939.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4939.png,Normal/images/Normal-4939.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4939.png,Normal/masks/Normal-4939.png,True,False,299,299,L,176.78,b8899fa5c4254c69e52f867e53fb9610
Normal-494.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-494.png,Normal/images/Normal-494.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-494.png,Normal/masks/Normal-494.png,True,False,299,299,L,109.81,cdb1d4e27cd733ab2ab1578510231075
Normal-4940.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4940.png,Normal/images/Normal-4940.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4940.png,Normal/masks/Normal-4940.png,True,False,299,299,L,126.3,c1204f62e56379b47436bd222a5d0518
Normal-4941.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4941.png,Normal/images/Normal-4941.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4941.png,Normal/masks/Normal-4941.png,True,False,299,299,L,182.22,79229facfeaba89bc590d8e49f5f5fe5
Normal-4942.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4942.png,Normal/images/Normal-4942.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4942.png,Normal/masks/Normal-4942.png,True,False,299,299,L,159.83,a5f586df673d1288efcc5767889dc8b9
Normal-4943.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4943.png,Normal/images/Normal-4943.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4943.png,Normal/masks/Normal-4943.png,True,False,299,299,L,102.68,1f516b02f88fd9e02b2e586f7a7bb23c
Normal-4944.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4944.png,Normal/images/Normal-4944.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4944.png,Normal/masks/Normal-4944.png,True,False,299,299,L,117.86,7f0ff42e59db4240dfd5979fc4747f6f
Normal-4945.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4945.png,Normal/images/Normal-4945.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4945.png,Normal/masks/Normal-4945.png,True,False,299,299,L,127.77,ebfe0a5737f2639376c615a408584ee1
Normal-4946.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4946.png,Normal/images/Normal-4946.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4946.png,Normal/masks/Normal-4946.png,True,False,299,299,L,127.97,5b26c820a3ee50bb6422d5cbd0f3a2fd
Normal-4947.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4947.png,Normal/images/Normal-4947.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4947.png,Normal/masks/Normal-4947.png,True,False,299,299,L,165.97,e64ae4069ab79df820b44eca6f84ba5d
Normal-4948.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4948.png,Normal/images/Normal-4948.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4948.png,Normal/masks/Normal-4948.png,True,False,299,299,L,126.14,fa3d6db3704fe9fc5250f1806a028e2a
Normal-4949.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4949.png,Normal/images/Normal-4949.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4949.png,Normal/masks/Normal-4949.png,True,False,299,299,L,133.17,7be211771ce1213edcc36218e2a9b7cb
Normal-495.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-495.png,Normal/images/Normal-495.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-495.png,Normal/masks/Normal-495.png,True,False,299,299,L,128.59,46bc4fd61f593bc1e350285c13cb632a
Normal-4950.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4950.png,Normal/images/Normal-4950.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4950.png,Normal/masks/Normal-4950.png,True,False,299,299,L,134.47,8527846cfcf782bca6719e8118018e50
Normal-4951.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4951.png,Normal/images/Normal-4951.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4951.png,Normal/masks/Normal-4951.png,True,False,299,299,L,123.95,c77e6745bd1c76c7024ce170503aae90
Normal-4952.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4952.png,Normal/images/Normal-4952.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4952.png,Normal/masks/Normal-4952.png,True,False,299,299,L,106.2,757f215762d57fe7a4588dbbea033cf2
Normal-4953.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4953.png,Normal/images/Normal-4953.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4953.png,Normal/masks/Normal-4953.png,True,False,299,299,L,163.11,2da132539e5c1269ae246182db2bf705
Normal-4954.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4954.png,Normal/images/Normal-4954.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4954.png,Normal/masks/Normal-4954.png,True,False,299,299,L,133.39,5c8bc543549bd3d4ee2fa8912f136b5c
Normal-4955.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4955.png,Normal/images/Normal-4955.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4955.png,Normal/masks/Normal-4955.png,True,False,299,299,L,128.22,4403050357576ef5a77ccdc56caa2220
Normal-4956.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4956.png,Normal/images/Normal-4956.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4956.png,Normal/masks/Normal-4956.png,True,False,299,299,L,166.7,553d1a7387fa9bf8c912b661c9e8758e
Normal-4957.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4957.png,Normal/images/Normal-4957.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4957.png,Normal/masks/Normal-4957.png,True,False,299,299,L,135.73,1cfbbaf622d7579b03923a4162b824cc
Normal-4958.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4958.png,Normal/images/Normal-4958.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4958.png,Normal/masks/Normal-4958.png,True,False,299,299,L,132.26,efd8b9df1f6b63eed00d4619c01c3494
Normal-4959.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4959.png,Normal/images/Normal-4959.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4959.png,Normal/masks/Normal-4959.png,True,False,299,299,L,102.58,94b337b56966dccb529f4c20ff5e50b5
Normal-496.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-496.png,Normal/images/Normal-496.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-496.png,Normal/masks/Normal-496.png,True,False,299,299,L,126.11,0ee24f0e514d6684ee136a0f9598f805
Normal-4960.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4960.png,Normal/images/Normal-4960.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4960.png,Normal/masks/Normal-4960.png,True,False,299,299,L,112.93,8598a77d22df4073343a88f747a513e6
Normal-4961.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4961.png,Normal/images/Normal-4961.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4961.png,Normal/masks/Normal-4961.png,True,False,299,299,L,138.38,596f4b4f1f9ab205a71aa1100a6600fb
Normal-4962.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4962.png,Normal/images/Normal-4962.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4962.png,Normal/masks/Normal-4962.png,True,False,299,299,L,118.61,278e3c0b31038ecee26c08099b3e021d
Normal-4963.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4963.png,Normal/images/Normal-4963.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4963.png,Normal/masks/Normal-4963.png,True,False,299,299,L,120.19,ababb5519d616e3014d7512baef3c136
Normal-4964.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4964.png,Normal/images/Normal-4964.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4964.png,Normal/masks/Normal-4964.png,True,False,299,299,L,106.79,c88fc1436aca83388496718927b3bb3e
Normal-4965.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4965.png,Normal/images/Normal-4965.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4965.png,Normal/masks/Normal-4965.png,True,False,299,299,L,125.66,9506ce651ebe2bbb5e293f085683fc22
Normal-4966.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4966.png,Normal/images/Normal-4966.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4966.png,Normal/masks/Normal-4966.png,True,False,299,299,L,116.19,f2c1a6190c175432e3862bec792153f7
Normal-4967.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4967.png,Normal/images/Normal-4967.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4967.png,Normal/masks/Normal-4967.png,True,False,299,299,L,173.55,8a00a0328c825fc23d28a1f5d295bf0d
Normal-4968.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4968.png,Normal/images/Normal-4968.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4968.png,Normal/masks/Normal-4968.png,True,False,299,299,L,138.06,09b9aac0eceadeaad3899af40aa32f27
Normal-4969.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4969.png,Normal/images/Normal-4969.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4969.png,Normal/masks/Normal-4969.png,True,False,299,299,L,107.84,36dde836bbcea1dbbd764bac1dfe8557
Normal-497.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-497.png,Normal/images/Normal-497.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-497.png,Normal/masks/Normal-497.png,True,False,299,299,L,143.12,d51de873a1c60218bccbe0fe230a8f3c
Normal-4970.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4970.png,Normal/images/Normal-4970.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4970.png,Normal/masks/Normal-4970.png,True,False,299,299,L,138.83,aaaed1c67743924d60161abd739bdca6
Normal-4971.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4971.png,Normal/images/Normal-4971.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4971.png,Normal/masks/Normal-4971.png,True,False,299,299,L,116.08,34463a6ed059acf4e6a6b485d5ab221a
Normal-4972.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4972.png,Normal/images/Normal-4972.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4972.png,Normal/masks/Normal-4972.png,True,False,299,299,L,104.23,12c61c73b45609ce64dca63217f48ffc
Normal-4973.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4973.png,Normal/images/Normal-4973.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4973.png,Normal/masks/Normal-4973.png,True,False,299,299,L,151.28,3db85d96f93877bb9181aaffa81b3e36
Normal-4974.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4974.png,Normal/images/Normal-4974.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4974.png,Normal/masks/Normal-4974.png,True,False,299,299,L,156.9,0f00bca912af8e7841fcfd949cec9e35
Normal-4975.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4975.png,Normal/images/Normal-4975.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4975.png,Normal/masks/Normal-4975.png,True,False,299,299,L,127.89,d9312a3e22a519e9353f4cee7e519b97
Normal-4976.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4976.png,Normal/images/Normal-4976.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4976.png,Normal/masks/Normal-4976.png,True,False,299,299,L,149.29,b812eee12c11116a61ff48d5eddca0b1
Normal-4977.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4977.png,Normal/images/Normal-4977.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4977.png,Normal/masks/Normal-4977.png,True,False,299,299,L,102.81,7970fc6846fe7bd4f6b243dfe3717514
Normal-4978.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4978.png,Normal/images/Normal-4978.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4978.png,Normal/masks/Normal-4978.png,True,False,299,299,L,87.75,9e5cd44ff272baed6456b30df2d5f829
Normal-4979.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4979.png,Normal/images/Normal-4979.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4979.png,Normal/masks/Normal-4979.png,True,False,299,299,L,128.48,c4acf2fce37f35144e724f4af9230df8
Normal-498.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-498.png,Normal/images/Normal-498.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-498.png,Normal/masks/Normal-498.png,True,False,299,299,L,122.48,92a1a35ae3031b902de4f7e1873f8721
Normal-4980.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4980.png,Normal/images/Normal-4980.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4980.png,Normal/masks/Normal-4980.png,True,False,299,299,L,186.05,0e0b5ee725ec3545acf522863898528f
Normal-4981.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4981.png,Normal/images/Normal-4981.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4981.png,Normal/masks/Normal-4981.png,True,False,299,299,L,173.63,3475e03225e2f7d77ce12493d673d356
Normal-4982.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4982.png,Normal/images/Normal-4982.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4982.png,Normal/masks/Normal-4982.png,True,False,299,299,L,118.95,9ce939ac6fde3dff2f171d7ab5944adf
Normal-4983.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4983.png,Normal/images/Normal-4983.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4983.png,Normal/masks/Normal-4983.png,True,False,299,299,L,129.8,a2822d3c26b34372ea218a30b6e6b899
Normal-4984.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4984.png,Normal/images/Normal-4984.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4984.png,Normal/masks/Normal-4984.png,True,False,299,299,L,103.69,ebf995f786e560d0a1ba29ce48cfbc2d
Normal-4985.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4985.png,Normal/images/Normal-4985.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4985.png,Normal/masks/Normal-4985.png,True,False,299,299,L,104.9,f9665bd84e710dfae14e2b414848cf40
Normal-4986.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4986.png,Normal/images/Normal-4986.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4986.png,Normal/masks/Normal-4986.png,True,False,299,299,L,167.88,919ec9644ad5221525d619e1d71f7d54
Normal-4987.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4987.png,Normal/images/Normal-4987.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4987.png,Normal/masks/Normal-4987.png,True,False,299,299,L,113.21,68f5bc4cf3aaf4ba07cbf4227094b8a8
Normal-4988.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4988.png,Normal/images/Normal-4988.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4988.png,Normal/masks/Normal-4988.png,True,False,299,299,L,178.99,4555e0b5441d084fe749258b222c1225
Normal-4989.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4989.png,Normal/images/Normal-4989.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4989.png,Normal/masks/Normal-4989.png,True,False,299,299,L,170.08,3c67a4e0be15281a848ec8be9a887ea3
Normal-499.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-499.png,Normal/images/Normal-499.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-499.png,Normal/masks/Normal-499.png,True,False,299,299,L,120.65,329fc955b4204842ec0fa1ba7366e12e
Normal-4990.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4990.png,Normal/images/Normal-4990.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4990.png,Normal/masks/Normal-4990.png,True,False,299,299,L,109.34,bf5481ce74200a85549418684a145cb7
Normal-4991.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4991.png,Normal/images/Normal-4991.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4991.png,Normal/masks/Normal-4991.png,True,False,299,299,L,149.45,4f858880749f15088614088369ff9991
Normal-4992.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4992.png,Normal/images/Normal-4992.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4992.png,Normal/masks/Normal-4992.png,True,False,299,299,L,149.82,f70f8353e9c9a9d9252e214d929854a2
Normal-4993.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4993.png,Normal/images/Normal-4993.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4993.png,Normal/masks/Normal-4993.png,True,False,299,299,L,108.56,77fe31351517d9ba9f2b0b06acafdd90
Normal-4994.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4994.png,Normal/images/Normal-4994.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4994.png,Normal/masks/Normal-4994.png,True,False,299,299,L,129.81,b8d27f823bea9d2a1e42a2e060a12d92
Normal-4995.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4995.png,Normal/images/Normal-4995.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4995.png,Normal/masks/Normal-4995.png,True,False,299,299,L,113.87,3c7375a2696614310e71040b3d4f8de7
Normal-4996.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4996.png,Normal/images/Normal-4996.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4996.png,Normal/masks/Normal-4996.png,True,False,299,299,L,105.22,69bb3bd5de04a8051971a3b5450b5bd2
Normal-4997.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4997.png,Normal/images/Normal-4997.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4997.png,Normal/masks/Normal-4997.png,True,False,299,299,L,118.9,6754f85ed05325d9b05a5871acd02086
Normal-4998.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4998.png,Normal/images/Normal-4998.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4998.png,Normal/masks/Normal-4998.png,True,False,299,299,L,171.71,b6d1e6fecfefb52c48ccf3a4ce87598d
Normal-4999.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-4999.png,Normal/images/Normal-4999.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-4999.png,Normal/masks/Normal-4999.png,True,False,299,299,L,100.29,e90c8a5d9a15a989d4c27976ea2490f7
Normal-5.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5.png,Normal/images/Normal-5.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5.png,Normal/masks/Normal-5.png,True,False,299,299,L,135.66,577c7ebb783331dd715bc9383ce3ab1c
Normal-50.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-50.png,Normal/images/Normal-50.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-50.png,Normal/masks/Normal-50.png,True,False,299,299,L,128.62,fd38d73681a57dbfa322b1e92178a868
Normal-500.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-500.png,Normal/images/Normal-500.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-500.png,Normal/masks/Normal-500.png,True,False,299,299,L,108.64,1e2bb1d37a42b0067fbf97f86b2cfb41
Normal-5000.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5000.png,Normal/images/Normal-5000.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5000.png,Normal/masks/Normal-5000.png,True,False,299,299,L,145.77,127dade236cd897c41e4b45c8906c1db
Normal-5001.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5001.png,Normal/images/Normal-5001.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5001.png,Normal/masks/Normal-5001.png,True,False,299,299,L,141.75,82cce4e9da32a5cccc030e6089e6a7bb
Normal-5002.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5002.png,Normal/images/Normal-5002.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5002.png,Normal/masks/Normal-5002.png,True,False,299,299,L,158.53,bd260aa402c08ac190f0216a7dd002e3
Normal-5003.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5003.png,Normal/images/Normal-5003.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5003.png,Normal/masks/Normal-5003.png,True,False,299,299,L,119.78,0bbc08b3dac5ee738fc3d0abeae6ca0c
Normal-5004.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5004.png,Normal/images/Normal-5004.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5004.png,Normal/masks/Normal-5004.png,True,False,299,299,L,114.03,8d6eab437dd6b7e9f709450a4ed18978
Normal-5005.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5005.png,Normal/images/Normal-5005.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5005.png,Normal/masks/Normal-5005.png,True,False,299,299,L,131.62,25d96ae9ab29ebdc021fd7723bff90ba
Normal-5006.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5006.png,Normal/images/Normal-5006.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5006.png,Normal/masks/Normal-5006.png,True,False,299,299,L,146.24,954e78ce6fc8c83ce59d9c61c49b6ee2
Normal-5007.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5007.png,Normal/images/Normal-5007.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5007.png,Normal/masks/Normal-5007.png,True,False,299,299,L,135.67,a351eb0f5ac853222332c23ae06faf0c
Normal-5008.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5008.png,Normal/images/Normal-5008.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5008.png,Normal/masks/Normal-5008.png,True,False,299,299,L,137.8,e8b05e297d0e3158203bed25463fcbef
Normal-5009.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5009.png,Normal/images/Normal-5009.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5009.png,Normal/masks/Normal-5009.png,True,False,299,299,L,150.1,e4b25b78b683dcefbe7e7a3d54f377cb
Normal-501.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-501.png,Normal/images/Normal-501.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-501.png,Normal/masks/Normal-501.png,True,False,299,299,L,110.19,290b66ccb97ff83fd38d90994d9c434e
Normal-5010.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5010.png,Normal/images/Normal-5010.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5010.png,Normal/masks/Normal-5010.png,True,False,299,299,L,130.25,2700b45a7d03ae29cd8f6d6e6fef9578
Normal-5011.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5011.png,Normal/images/Normal-5011.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5011.png,Normal/masks/Normal-5011.png,True,False,299,299,L,103.57,f17c358fcdfd811a3a07a931a9be8cd1
Normal-5012.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5012.png,Normal/images/Normal-5012.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5012.png,Normal/masks/Normal-5012.png,True,False,299,299,L,112.04,aa5a66bb7ceb412fd793944fcdbd5f9d
Normal-5013.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5013.png,Normal/images/Normal-5013.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5013.png,Normal/masks/Normal-5013.png,True,False,299,299,L,122.5,207f088935f416077322ca279e787956
Normal-5014.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5014.png,Normal/images/Normal-5014.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5014.png,Normal/masks/Normal-5014.png,True,False,299,299,L,170.36,7d53775e1099e734621addde615ddca1
Normal-5015.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5015.png,Normal/images/Normal-5015.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5015.png,Normal/masks/Normal-5015.png,True,False,299,299,L,138.75,a01dd4f1f8ebf72000485fabcdb65a29
Normal-5016.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5016.png,Normal/images/Normal-5016.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5016.png,Normal/masks/Normal-5016.png,True,False,299,299,L,137.73,5505e653a4d89ef65efa5af62168003d
Normal-5017.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5017.png,Normal/images/Normal-5017.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5017.png,Normal/masks/Normal-5017.png,True,False,299,299,L,112.11,23a3c6c0d0912c2183700222d325cf49
Normal-5018.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5018.png,Normal/images/Normal-5018.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5018.png,Normal/masks/Normal-5018.png,True,False,299,299,L,145.98,2aef2f8f857d6cac549512d41c11badd
Normal-5019.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5019.png,Normal/images/Normal-5019.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5019.png,Normal/masks/Normal-5019.png,True,False,299,299,L,171.08,ed4d95b1fb6ee7feccafa842c9f2966b
Normal-502.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-502.png,Normal/images/Normal-502.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-502.png,Normal/masks/Normal-502.png,True,False,299,299,L,118.3,0548148a3ae6daa1bffab2a2f411e69b
Normal-5020.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5020.png,Normal/images/Normal-5020.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5020.png,Normal/masks/Normal-5020.png,True,False,299,299,L,107.6,cea8e3ed922f996b52e51774eb46f251
Normal-5021.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5021.png,Normal/images/Normal-5021.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5021.png,Normal/masks/Normal-5021.png,True,False,299,299,L,135.16,0f1d85942ae1894ccccd590b81a5cd34
Normal-5022.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5022.png,Normal/images/Normal-5022.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5022.png,Normal/masks/Normal-5022.png,True,False,299,299,L,140.98,237faf511a602ce08d327b67f183c09f
Normal-5023.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5023.png,Normal/images/Normal-5023.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5023.png,Normal/masks/Normal-5023.png,True,False,299,299,L,159.95,771427e8b8effa18c71f8a788639f3d5
Normal-5024.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5024.png,Normal/images/Normal-5024.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5024.png,Normal/masks/Normal-5024.png,True,False,299,299,L,154.57,d57302ec8e32612f985dca3ab9526dec
Normal-5025.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5025.png,Normal/images/Normal-5025.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5025.png,Normal/masks/Normal-5025.png,True,False,299,299,L,121.09,e2dd63e405fede87727948fdc2aaeb2e
Normal-5026.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5026.png,Normal/images/Normal-5026.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5026.png,Normal/masks/Normal-5026.png,True,False,299,299,L,140.43,d75a1eeabf3b61f98c2aee1fee5b8932
Normal-5027.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5027.png,Normal/images/Normal-5027.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5027.png,Normal/masks/Normal-5027.png,True,False,299,299,L,146.05,b2efcd05510262c29e69d182cb140927
Normal-5028.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5028.png,Normal/images/Normal-5028.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5028.png,Normal/masks/Normal-5028.png,True,False,299,299,L,163.98,20215d32aeebcbbd90464f103d85c78c
Normal-5029.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5029.png,Normal/images/Normal-5029.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5029.png,Normal/masks/Normal-5029.png,True,False,299,299,L,123.38,6ff9f2104bfccf1e11493d12955014f7
Normal-503.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-503.png,Normal/images/Normal-503.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-503.png,Normal/masks/Normal-503.png,True,False,299,299,L,109.94,10d0f221528fe210b7d9ceb8ebfce9f7
Normal-5030.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5030.png,Normal/images/Normal-5030.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5030.png,Normal/masks/Normal-5030.png,True,False,299,299,L,162.14,e155520ef72a529838800439a8860a93
Normal-5031.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5031.png,Normal/images/Normal-5031.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5031.png,Normal/masks/Normal-5031.png,True,False,299,299,L,96.95,0dee4de682b9a89b285a8571ba6134da
Normal-5032.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5032.png,Normal/images/Normal-5032.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5032.png,Normal/masks/Normal-5032.png,True,False,299,299,L,113.62,88a74eb5a4c98919c15283500ae8a408
Normal-5033.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5033.png,Normal/images/Normal-5033.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5033.png,Normal/masks/Normal-5033.png,True,False,299,299,L,166.46,356c378849a2f2b5a9c05b83ee178443
Normal-5034.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5034.png,Normal/images/Normal-5034.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5034.png,Normal/masks/Normal-5034.png,True,False,299,299,L,135.45,f3b9c19c7c0c502169c1483558c35f71
Normal-5035.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5035.png,Normal/images/Normal-5035.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5035.png,Normal/masks/Normal-5035.png,True,False,299,299,L,109.16,65ef827c1ed7a1d4b0141754ba22f5c7
Normal-5036.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5036.png,Normal/images/Normal-5036.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5036.png,Normal/masks/Normal-5036.png,True,False,299,299,L,109.95,13908988033a2e6a73c171c1e6ab1cb3
Normal-5037.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5037.png,Normal/images/Normal-5037.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5037.png,Normal/masks/Normal-5037.png,True,False,299,299,L,115.21,36f9a5e5704cb825156f1128694ff2ca
Normal-5038.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5038.png,Normal/images/Normal-5038.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5038.png,Normal/masks/Normal-5038.png,True,False,299,299,L,172.6,295a6606360ae310c893142f92407083
Normal-5039.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5039.png,Normal/images/Normal-5039.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5039.png,Normal/masks/Normal-5039.png,True,False,299,299,L,106.67,b231943d1e8901df3a620785696ea69c
Normal-504.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-504.png,Normal/images/Normal-504.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-504.png,Normal/masks/Normal-504.png,True,False,299,299,L,106.88,a9713ef6cd764c5ae5a5336995c31ab9
Normal-5040.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5040.png,Normal/images/Normal-5040.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5040.png,Normal/masks/Normal-5040.png,True,False,299,299,L,172.2,2baf18138fe642493c5b12f12e97ece1
Normal-5041.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5041.png,Normal/images/Normal-5041.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5041.png,Normal/masks/Normal-5041.png,True,False,299,299,L,109.31,1d4a3044f3b5151a811f37af45395848
Normal-5042.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5042.png,Normal/images/Normal-5042.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5042.png,Normal/masks/Normal-5042.png,True,False,299,299,L,85.97,b40d5f9776c402ec54318d4f04f84a88
Normal-5043.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5043.png,Normal/images/Normal-5043.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5043.png,Normal/masks/Normal-5043.png,True,False,299,299,L,150.33,42855f0fb771ed8ba5393e043b842f65
Normal-5044.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5044.png,Normal/images/Normal-5044.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5044.png,Normal/masks/Normal-5044.png,True,False,299,299,L,143.11,0a7b6ed021e8cb6d3db11c4b86dedccf
Normal-5045.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5045.png,Normal/images/Normal-5045.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5045.png,Normal/masks/Normal-5045.png,True,False,299,299,L,170.58,20d309b8587eb525eff59d946d6478f6
Normal-5046.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5046.png,Normal/images/Normal-5046.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5046.png,Normal/masks/Normal-5046.png,True,False,299,299,L,69.35,c880d15826b362648d44de52fd73930a
Normal-5047.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5047.png,Normal/images/Normal-5047.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5047.png,Normal/masks/Normal-5047.png,True,False,299,299,L,114.76,9a4e9153c7a962e4662b349054f63fd9
Normal-5048.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5048.png,Normal/images/Normal-5048.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5048.png,Normal/masks/Normal-5048.png,True,False,299,299,L,166.8,af8971e0b61f62dfa60a44f3072f7621
Normal-5049.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5049.png,Normal/images/Normal-5049.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5049.png,Normal/masks/Normal-5049.png,True,False,299,299,L,102.57,b1f4c4fafbf46802cd469ee5d8cac0c5
Normal-505.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-505.png,Normal/images/Normal-505.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-505.png,Normal/masks/Normal-505.png,True,False,299,299,L,126.05,d9c1ca704b4455b3f9c4614d4cecd84e
Normal-5050.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5050.png,Normal/images/Normal-5050.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5050.png,Normal/masks/Normal-5050.png,True,False,299,299,L,126.71,690e4bd6343dce858d7f453807cf66c8
Normal-5051.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5051.png,Normal/images/Normal-5051.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5051.png,Normal/masks/Normal-5051.png,True,False,299,299,L,133.53,d25917a89b11bfde27dc5cb082cb3512
Normal-5052.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5052.png,Normal/images/Normal-5052.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5052.png,Normal/masks/Normal-5052.png,True,False,299,299,L,120.44,b47f62d40f8e2e944e364fe1b548c61e
Normal-5053.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5053.png,Normal/images/Normal-5053.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5053.png,Normal/masks/Normal-5053.png,True,False,299,299,L,171.81,be0b851836dbfa770d394900103a40ef
Normal-5054.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5054.png,Normal/images/Normal-5054.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5054.png,Normal/masks/Normal-5054.png,True,False,299,299,L,129.66,3d08c75b6c9a860a0fc1018f4fdd96fc
Normal-5055.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5055.png,Normal/images/Normal-5055.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5055.png,Normal/masks/Normal-5055.png,True,False,299,299,L,103.03,a76744887653f459aa30a93c973c9c04
Normal-5056.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5056.png,Normal/images/Normal-5056.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5056.png,Normal/masks/Normal-5056.png,True,False,299,299,L,121.59,dc5d22c1506b714004c7e97d3f167a4f
Normal-5057.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5057.png,Normal/images/Normal-5057.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5057.png,Normal/masks/Normal-5057.png,True,False,299,299,L,117.6,50332d44c8ee9d79c6ed11a8f737f42c
Normal-5058.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5058.png,Normal/images/Normal-5058.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5058.png,Normal/masks/Normal-5058.png,True,False,299,299,L,120.01,df9f80d097e25e66a8c12c9d3bb0ec6c
Normal-5059.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5059.png,Normal/images/Normal-5059.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5059.png,Normal/masks/Normal-5059.png,True,False,299,299,L,96.94,4496e55803b1490fb652e498cbc954a8
Normal-506.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-506.png,Normal/images/Normal-506.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-506.png,Normal/masks/Normal-506.png,True,False,299,299,L,109.54,49a7f878f76c44efbac656e85a3315d2
Normal-5060.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5060.png,Normal/images/Normal-5060.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5060.png,Normal/masks/Normal-5060.png,True,False,299,299,L,174.88,74288fb9d749e2c525981df72b31304a
Normal-5061.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5061.png,Normal/images/Normal-5061.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5061.png,Normal/masks/Normal-5061.png,True,False,299,299,L,156.74,58660a61dafafb5d68b754641aa90ceb
Normal-5062.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5062.png,Normal/images/Normal-5062.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5062.png,Normal/masks/Normal-5062.png,True,False,299,299,L,141.57,90056e7c31c85560280ddfc5f7582271
Normal-5063.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5063.png,Normal/images/Normal-5063.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5063.png,Normal/masks/Normal-5063.png,True,False,299,299,L,106.68,d3d93a0460491b52c3213c7eec8b4845
Normal-5064.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5064.png,Normal/images/Normal-5064.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5064.png,Normal/masks/Normal-5064.png,True,False,299,299,L,156.16,5edaeaf2f33dc0b800025e2c7f61d84a
Normal-5065.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5065.png,Normal/images/Normal-5065.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5065.png,Normal/masks/Normal-5065.png,True,False,299,299,L,161.16,7841fe733cbb711ebf7cbbbda6b62264
Normal-5066.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5066.png,Normal/images/Normal-5066.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5066.png,Normal/masks/Normal-5066.png,True,False,299,299,L,111.73,3ea15a6db2f4f383cb6a2342622b3c38
Normal-5067.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5067.png,Normal/images/Normal-5067.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5067.png,Normal/masks/Normal-5067.png,True,False,299,299,L,168.08,7111a8305170854d9c3ed818b5e82873
Normal-5068.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5068.png,Normal/images/Normal-5068.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5068.png,Normal/masks/Normal-5068.png,True,False,299,299,L,128.65,07ecb619c2651367ce3e07447516131d
Normal-5069.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5069.png,Normal/images/Normal-5069.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5069.png,Normal/masks/Normal-5069.png,True,False,299,299,L,71.99,1b1e834f7c8821141a229f3d2cc86d08
Normal-507.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-507.png,Normal/images/Normal-507.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-507.png,Normal/masks/Normal-507.png,True,False,299,299,L,125.27,4e0bed029bd4ee0ec324742e198cc9f8
Normal-5070.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5070.png,Normal/images/Normal-5070.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5070.png,Normal/masks/Normal-5070.png,True,False,299,299,L,105.77,f05b913caf5ef837b9c1488a716b8545
Normal-5071.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5071.png,Normal/images/Normal-5071.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5071.png,Normal/masks/Normal-5071.png,True,False,299,299,L,129.12,c7b08299ea1d362130832274c0e9ad39
Normal-5072.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5072.png,Normal/images/Normal-5072.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5072.png,Normal/masks/Normal-5072.png,True,False,299,299,L,116.05,71a5c497306f54d08fa0067911ab28e2
Normal-5073.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5073.png,Normal/images/Normal-5073.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5073.png,Normal/masks/Normal-5073.png,True,False,299,299,L,153.34,64ae156fbd002e4bfc05c7d80b7685eb
Normal-5074.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5074.png,Normal/images/Normal-5074.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5074.png,Normal/masks/Normal-5074.png,True,False,299,299,L,142.06,1f814dd5392d411edd33fb23081d681b
Normal-5075.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5075.png,Normal/images/Normal-5075.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5075.png,Normal/masks/Normal-5075.png,True,False,299,299,L,178.2,0956c101f0571044d76135a478f39582
Normal-5076.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5076.png,Normal/images/Normal-5076.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5076.png,Normal/masks/Normal-5076.png,True,False,299,299,L,124.66,3e2fe9f3b2c9f8cae16afc79841d9312
Normal-5077.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5077.png,Normal/images/Normal-5077.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5077.png,Normal/masks/Normal-5077.png,True,False,299,299,L,147.23,aba29b5beb5764d3bf70edd008769567
Normal-5078.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5078.png,Normal/images/Normal-5078.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5078.png,Normal/masks/Normal-5078.png,True,False,299,299,L,130.07,38a3a116e7257a7db2660e72cd5fe6fa
Normal-5079.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5079.png,Normal/images/Normal-5079.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5079.png,Normal/masks/Normal-5079.png,True,False,299,299,L,152.57,074087ba76b5163260676d8e0d7a0190
Normal-508.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-508.png,Normal/images/Normal-508.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-508.png,Normal/masks/Normal-508.png,True,False,299,299,L,106.2,9141b96abdfb10bb1703b8e928212448
Normal-5080.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5080.png,Normal/images/Normal-5080.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5080.png,Normal/masks/Normal-5080.png,True,False,299,299,L,147.73,40f3776a8d79352af16845842545ad86
Normal-5081.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5081.png,Normal/images/Normal-5081.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5081.png,Normal/masks/Normal-5081.png,True,False,299,299,L,153.01,4f54b3aa5197b9b32b2bc7dbd91a6112
Normal-5082.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5082.png,Normal/images/Normal-5082.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5082.png,Normal/masks/Normal-5082.png,True,False,299,299,L,110.53,8b2ef52feb4bac7a445b18eb6af87dc7
Normal-5083.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5083.png,Normal/images/Normal-5083.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5083.png,Normal/masks/Normal-5083.png,True,False,299,299,L,155.77,09b1586617652cacc777c05bfea520ee
Normal-5084.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5084.png,Normal/images/Normal-5084.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5084.png,Normal/masks/Normal-5084.png,True,False,299,299,L,174.63,f727cbe9536db5c0a46c3e9bb769c7f7
Normal-5085.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5085.png,Normal/images/Normal-5085.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5085.png,Normal/masks/Normal-5085.png,True,False,299,299,L,142.25,eea2becf40205951477fd420c0996876
Normal-5086.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5086.png,Normal/images/Normal-5086.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5086.png,Normal/masks/Normal-5086.png,True,False,299,299,L,121.27,13a71ae3259d8fcf1986a90c23c6f6da
Normal-5087.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5087.png,Normal/images/Normal-5087.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5087.png,Normal/masks/Normal-5087.png,True,False,299,299,L,164.11,4f444720c85aa13f46a70855e4bb445b
Normal-5088.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5088.png,Normal/images/Normal-5088.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5088.png,Normal/masks/Normal-5088.png,True,False,299,299,L,70.26,920177e2f1f357b0900d220ccb6af89a
Normal-5089.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5089.png,Normal/images/Normal-5089.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5089.png,Normal/masks/Normal-5089.png,True,False,299,299,L,125.76,48e8e594d1801631f68109ce7fff9b97
Normal-509.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-509.png,Normal/images/Normal-509.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-509.png,Normal/masks/Normal-509.png,True,False,299,299,L,109.21,95a94b691a57d6321abb1e592416c3e7
Normal-5090.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5090.png,Normal/images/Normal-5090.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5090.png,Normal/masks/Normal-5090.png,True,False,299,299,L,98.12,05577e37fc566320cf9263f3830fd841
Normal-5091.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5091.png,Normal/images/Normal-5091.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5091.png,Normal/masks/Normal-5091.png,True,False,299,299,L,114.89,3c27087a4e4fdce4fa94f847db3d04ea
Normal-5092.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5092.png,Normal/images/Normal-5092.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5092.png,Normal/masks/Normal-5092.png,True,False,299,299,L,172.84,98b9cf4d2fe6944b68fdc35ba1b934de
Normal-5093.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5093.png,Normal/images/Normal-5093.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5093.png,Normal/masks/Normal-5093.png,True,False,299,299,L,127.95,a7db3c276063e7ba95a4118f4d494d03
Normal-5094.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5094.png,Normal/images/Normal-5094.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5094.png,Normal/masks/Normal-5094.png,True,False,299,299,L,115.64,c3a31b3b2ebed8d8b714829be964cf70
Normal-5095.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5095.png,Normal/images/Normal-5095.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5095.png,Normal/masks/Normal-5095.png,True,False,299,299,L,127.96,54ac5845a50c14857b1b187d3e6a6434
Normal-5096.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5096.png,Normal/images/Normal-5096.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5096.png,Normal/masks/Normal-5096.png,True,False,299,299,L,130.55,12a1444f59dc4a813fdffabd2f91579b
Normal-5097.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5097.png,Normal/images/Normal-5097.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5097.png,Normal/masks/Normal-5097.png,True,False,299,299,L,142.2,e7cf0a528bfb67ad71151ac37da09968
Normal-5098.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5098.png,Normal/images/Normal-5098.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5098.png,Normal/masks/Normal-5098.png,True,False,299,299,L,145.18,928920a1e77439f231dabc793dd2fc24
Normal-5099.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5099.png,Normal/images/Normal-5099.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5099.png,Normal/masks/Normal-5099.png,True,False,299,299,L,120.04,dd1de14d3716fa1fc8b4b04b92bc4e5f
Normal-51.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-51.png,Normal/images/Normal-51.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-51.png,Normal/masks/Normal-51.png,True,False,299,299,L,123.99,8774fe287a8a8bdd62b0b269611b1ae1
Normal-510.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-510.png,Normal/images/Normal-510.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-510.png,Normal/masks/Normal-510.png,True,False,299,299,L,138.33,57a9929566242892b731994aacf14028
Normal-5100.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5100.png,Normal/images/Normal-5100.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5100.png,Normal/masks/Normal-5100.png,True,False,299,299,L,113.33,3cf48327dc3c660812975307bbe5800a
Normal-5101.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5101.png,Normal/images/Normal-5101.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5101.png,Normal/masks/Normal-5101.png,True,False,299,299,L,124.39,af358624db2e566ae2f9f6723a5031e7
Normal-5102.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5102.png,Normal/images/Normal-5102.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5102.png,Normal/masks/Normal-5102.png,True,False,299,299,L,142.25,fc0a7a82f68fbcbfff5fc0895e544b77
Normal-5103.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5103.png,Normal/images/Normal-5103.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5103.png,Normal/masks/Normal-5103.png,True,False,299,299,L,114.77,992bad65f8f8801692908cbefeb99d2b
Normal-5104.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5104.png,Normal/images/Normal-5104.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5104.png,Normal/masks/Normal-5104.png,True,False,299,299,L,113.85,ce1d09fc08612d98f287f496fed65c5e
Normal-5105.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5105.png,Normal/images/Normal-5105.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5105.png,Normal/masks/Normal-5105.png,True,False,299,299,L,117.39,92556f12868614cfd6bc8acef37d32b7
Normal-5106.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5106.png,Normal/images/Normal-5106.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5106.png,Normal/masks/Normal-5106.png,True,False,299,299,L,121.54,5e0699d393587d712de64e5e87ee333a
Normal-5107.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5107.png,Normal/images/Normal-5107.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5107.png,Normal/masks/Normal-5107.png,True,False,299,299,L,117.93,9b95aa234340a2ef2a79e09d8a106a31
Normal-5108.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5108.png,Normal/images/Normal-5108.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5108.png,Normal/masks/Normal-5108.png,True,False,299,299,L,94.92,e91ef7316935b42f4384332a845d5554
Normal-5109.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5109.png,Normal/images/Normal-5109.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5109.png,Normal/masks/Normal-5109.png,True,False,299,299,L,110.62,4caf96b3759257b1186e566a73c8fe62
Normal-511.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-511.png,Normal/images/Normal-511.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-511.png,Normal/masks/Normal-511.png,True,False,299,299,L,126.63,202343589d0a1695870f47259d952544
Normal-5110.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5110.png,Normal/images/Normal-5110.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5110.png,Normal/masks/Normal-5110.png,True,False,299,299,L,106.31,7d44a7a0312c4c26f1c5e4e55616669d
Normal-5111.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5111.png,Normal/images/Normal-5111.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5111.png,Normal/masks/Normal-5111.png,True,False,299,299,L,133.51,aa340faddbf9c190747b8f68c8a877bc
Normal-5112.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5112.png,Normal/images/Normal-5112.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5112.png,Normal/masks/Normal-5112.png,True,False,299,299,L,154.16,1733483eb25c4c3f5ce0cbbee80f0c22
Normal-5113.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5113.png,Normal/images/Normal-5113.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5113.png,Normal/masks/Normal-5113.png,True,False,299,299,L,119.26,155b5fa5e7e66e1469be1bf59d813a50
Normal-5114.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5114.png,Normal/images/Normal-5114.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5114.png,Normal/masks/Normal-5114.png,True,False,299,299,L,123.99,515ea49e82f12ec4b02c642c85e7414f
Normal-5115.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5115.png,Normal/images/Normal-5115.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5115.png,Normal/masks/Normal-5115.png,True,False,299,299,L,130.2,77ec3d1e1a27c73b4be98d6c2c924615
Normal-5116.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5116.png,Normal/images/Normal-5116.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5116.png,Normal/masks/Normal-5116.png,True,False,299,299,L,171.57,face98ad613fff24a80d0a68b2497f20
Normal-5117.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5117.png,Normal/images/Normal-5117.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5117.png,Normal/masks/Normal-5117.png,True,False,299,299,L,113.8,9dcf552ae447bf0413b395596b5b6f38
Normal-5118.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5118.png,Normal/images/Normal-5118.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5118.png,Normal/masks/Normal-5118.png,True,False,299,299,L,132.09,d90206538ff46360435ecea24b98f3e0
Normal-5119.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5119.png,Normal/images/Normal-5119.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5119.png,Normal/masks/Normal-5119.png,True,False,299,299,L,114.58,c6a4d2c088b9ebd21f5f5a65bc764545
Normal-512.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-512.png,Normal/images/Normal-512.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-512.png,Normal/masks/Normal-512.png,True,False,299,299,L,129.22,a89736fe3b0fd73e27e8f6100c2aca15
Normal-5120.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5120.png,Normal/images/Normal-5120.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5120.png,Normal/masks/Normal-5120.png,True,False,299,299,L,166.68,d0031accf51207a7e57d0393e2b589ee
Normal-5121.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5121.png,Normal/images/Normal-5121.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5121.png,Normal/masks/Normal-5121.png,True,False,299,299,L,99.85,0e35d7ec26d922a75495adf5fc5016fd
Normal-5122.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5122.png,Normal/images/Normal-5122.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5122.png,Normal/masks/Normal-5122.png,True,False,299,299,L,108.63,b28b6392d735eefb9c97bb3c4bc1e467
Normal-5123.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5123.png,Normal/images/Normal-5123.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5123.png,Normal/masks/Normal-5123.png,True,False,299,299,L,130.11,5d4fe0551555759745e65677356cdec2
Normal-5124.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5124.png,Normal/images/Normal-5124.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5124.png,Normal/masks/Normal-5124.png,True,False,299,299,L,113.88,58b16d2a6fe3de4df5cd0912d6680c3b
Normal-5125.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5125.png,Normal/images/Normal-5125.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5125.png,Normal/masks/Normal-5125.png,True,False,299,299,L,120.74,c213ff9c367a595ff68e7708ae188772
Normal-5126.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5126.png,Normal/images/Normal-5126.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5126.png,Normal/masks/Normal-5126.png,True,False,299,299,L,149.85,53981a49a9ed7831b6c068b0a538a65a
Normal-5127.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5127.png,Normal/images/Normal-5127.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5127.png,Normal/masks/Normal-5127.png,True,False,299,299,L,97.52,9663e43e20a275e81aa8b8fa17a3af7c
Normal-5128.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5128.png,Normal/images/Normal-5128.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5128.png,Normal/masks/Normal-5128.png,True,False,299,299,L,136.36,4ac3364af55f0ec91bf39a25e127b7ac
Normal-5129.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5129.png,Normal/images/Normal-5129.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5129.png,Normal/masks/Normal-5129.png,True,False,299,299,L,112.28,48accbf2ffc3248106785aca1089d62d
Normal-513.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-513.png,Normal/images/Normal-513.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-513.png,Normal/masks/Normal-513.png,True,False,299,299,L,131.62,313e75545aa1570f992e816d3f200d37
Normal-5130.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5130.png,Normal/images/Normal-5130.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5130.png,Normal/masks/Normal-5130.png,True,False,299,299,L,116.76,ba71d3284ff326948a2eb262d8add638
Normal-5131.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5131.png,Normal/images/Normal-5131.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5131.png,Normal/masks/Normal-5131.png,True,False,299,299,L,100.8,627b59872a6e28d556f882dd816f8c62
Normal-5132.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5132.png,Normal/images/Normal-5132.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5132.png,Normal/masks/Normal-5132.png,True,False,299,299,L,133.78,1c23baed7033808172127817ac09f184
Normal-5133.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5133.png,Normal/images/Normal-5133.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5133.png,Normal/masks/Normal-5133.png,True,False,299,299,L,134.71,363c30ff73b51dc06c97581948ffafce
Normal-5134.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5134.png,Normal/images/Normal-5134.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5134.png,Normal/masks/Normal-5134.png,True,False,299,299,L,128.83,480c81d998758af0c232012deb714c9b
Normal-5135.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5135.png,Normal/images/Normal-5135.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5135.png,Normal/masks/Normal-5135.png,True,False,299,299,L,166.7,43cc5b474f25e7b3c4b8bff23772fff4
Normal-5136.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5136.png,Normal/images/Normal-5136.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5136.png,Normal/masks/Normal-5136.png,True,False,299,299,L,150.74,75c406bba4d6320415b607734c8f9eb0
Normal-5137.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5137.png,Normal/images/Normal-5137.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5137.png,Normal/masks/Normal-5137.png,True,False,299,299,L,122.86,921c273250b03dde71cef9368b7576a8
Normal-5138.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5138.png,Normal/images/Normal-5138.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5138.png,Normal/masks/Normal-5138.png,True,False,299,299,L,127.76,f73461c6f379b8eed501adc7db53db0e
Normal-5139.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5139.png,Normal/images/Normal-5139.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5139.png,Normal/masks/Normal-5139.png,True,False,299,299,L,164.51,e1a763b6bf5d614ba67dd3a8ab78ae84
Normal-514.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-514.png,Normal/images/Normal-514.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-514.png,Normal/masks/Normal-514.png,True,False,299,299,L,91.7,fea82931c5a83b1866acc1cfb261ad61
Normal-5140.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5140.png,Normal/images/Normal-5140.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5140.png,Normal/masks/Normal-5140.png,True,False,299,299,L,123.17,43cba86c7f34ec6fe808bca70a4fbbc9
Normal-5141.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5141.png,Normal/images/Normal-5141.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5141.png,Normal/masks/Normal-5141.png,True,False,299,299,L,152.96,42812414d6d70c552223b8f0cf404377
Normal-5142.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5142.png,Normal/images/Normal-5142.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5142.png,Normal/masks/Normal-5142.png,True,False,299,299,L,137.9,5787d01f45dfa8f11e1dd267540018a0
Normal-5143.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5143.png,Normal/images/Normal-5143.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5143.png,Normal/masks/Normal-5143.png,True,False,299,299,L,126.82,5df720116cec38d03a38d9f047caad98
Normal-5144.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5144.png,Normal/images/Normal-5144.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5144.png,Normal/masks/Normal-5144.png,True,False,299,299,L,117.68,ba755043d2472cabe2e95e8d61de370a
Normal-5145.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5145.png,Normal/images/Normal-5145.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5145.png,Normal/masks/Normal-5145.png,True,False,299,299,L,185.68,e0d19f93550b186723f813f47348de03
Normal-5146.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5146.png,Normal/images/Normal-5146.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5146.png,Normal/masks/Normal-5146.png,True,False,299,299,L,165.23,856c6667f46cc1d5a20dde94b0c4e056
Normal-5147.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5147.png,Normal/images/Normal-5147.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5147.png,Normal/masks/Normal-5147.png,True,False,299,299,L,154.82,08df7a7039d37c45c740d01595440167
Normal-5148.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5148.png,Normal/images/Normal-5148.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5148.png,Normal/masks/Normal-5148.png,True,False,299,299,L,120.11,82cdef71748250b2aa96ecb30089becd
Normal-5149.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5149.png,Normal/images/Normal-5149.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5149.png,Normal/masks/Normal-5149.png,True,False,299,299,L,106.65,8f4288138542e66bfc1d43cdd72801bd
Normal-515.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-515.png,Normal/images/Normal-515.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-515.png,Normal/masks/Normal-515.png,True,False,299,299,L,114.11,658759c5d819f4626aef4ea261e2c465
Normal-5150.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5150.png,Normal/images/Normal-5150.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5150.png,Normal/masks/Normal-5150.png,True,False,299,299,L,123.08,62921b3c7cb6d71aae1d28907f6dce9f
Normal-5151.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5151.png,Normal/images/Normal-5151.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5151.png,Normal/masks/Normal-5151.png,True,False,299,299,L,113.2,ace4ec776f3711d9ccf09343f7ae3572
Normal-5152.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5152.png,Normal/images/Normal-5152.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5152.png,Normal/masks/Normal-5152.png,True,False,299,299,L,126.8,7ec176360c94e639331f7fa3d690e865
Normal-5153.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5153.png,Normal/images/Normal-5153.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5153.png,Normal/masks/Normal-5153.png,True,False,299,299,L,112.07,b0b72dbedb992a2b62db411df800fd9c
Normal-5154.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5154.png,Normal/images/Normal-5154.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5154.png,Normal/masks/Normal-5154.png,True,False,299,299,L,112.15,242cb783db411e26b7c49264d49dfe51
Normal-5155.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5155.png,Normal/images/Normal-5155.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5155.png,Normal/masks/Normal-5155.png,True,False,299,299,L,170.39,90749ab1d539a058ccf794b07dbb119c
Normal-5156.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5156.png,Normal/images/Normal-5156.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5156.png,Normal/masks/Normal-5156.png,True,False,299,299,L,106.11,a5559124c6e86182cce5659fbbe4c6db
Normal-5157.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5157.png,Normal/images/Normal-5157.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5157.png,Normal/masks/Normal-5157.png,True,False,299,299,L,103.76,f8c7250c38ba37954da89cf822549522
Normal-5158.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5158.png,Normal/images/Normal-5158.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5158.png,Normal/masks/Normal-5158.png,True,False,299,299,L,106.78,68da04ce57316552357b0b228a7fedde
Normal-5159.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5159.png,Normal/images/Normal-5159.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5159.png,Normal/masks/Normal-5159.png,True,False,299,299,L,142.42,07459c94993aa0cf50c2aef5c6db81d1
Normal-516.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-516.png,Normal/images/Normal-516.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-516.png,Normal/masks/Normal-516.png,True,False,299,299,L,119.15,2fdcd680f9c7b6b545bdbde2e2d9c2d8
Normal-5160.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5160.png,Normal/images/Normal-5160.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5160.png,Normal/masks/Normal-5160.png,True,False,299,299,L,151.41,9b074665d76f1e89241fb5b22f40388d
Normal-5161.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5161.png,Normal/images/Normal-5161.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5161.png,Normal/masks/Normal-5161.png,True,False,299,299,L,119.92,e29ccc78a70b72b9fa7d3fe3700d3d0d
Normal-5162.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5162.png,Normal/images/Normal-5162.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5162.png,Normal/masks/Normal-5162.png,True,False,299,299,L,104.09,65250de3bceb65dba80ce96d2cc4b7e6
Normal-5163.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5163.png,Normal/images/Normal-5163.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5163.png,Normal/masks/Normal-5163.png,True,False,299,299,L,126.47,176838d4abf02eeff9847d00534ef587
Normal-5164.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5164.png,Normal/images/Normal-5164.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5164.png,Normal/masks/Normal-5164.png,True,False,299,299,L,171.19,918abffa382d6c409b67c20d06d88420
Normal-5165.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5165.png,Normal/images/Normal-5165.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5165.png,Normal/masks/Normal-5165.png,True,False,299,299,L,99.58,39e8e26e29a0fc789ec7234e3ead928f
Normal-5166.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5166.png,Normal/images/Normal-5166.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5166.png,Normal/masks/Normal-5166.png,True,False,299,299,L,146.31,807370677a7b84f663ccf2825c77b461
Normal-5167.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5167.png,Normal/images/Normal-5167.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5167.png,Normal/masks/Normal-5167.png,True,False,299,299,L,98.1,360463318e71e4e5e7a7d92dc30a6bc5
Normal-5168.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5168.png,Normal/images/Normal-5168.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5168.png,Normal/masks/Normal-5168.png,True,False,299,299,L,160.29,10aff4cef26540f8a964e7e463d03a72
Normal-5169.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5169.png,Normal/images/Normal-5169.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5169.png,Normal/masks/Normal-5169.png,True,False,299,299,L,121.6,d23011b269f0f5ffcf34cbe719bcea23
Normal-517.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-517.png,Normal/images/Normal-517.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-517.png,Normal/masks/Normal-517.png,True,False,299,299,L,118.73,987671c03742262b186790384016ef1c
Normal-5170.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5170.png,Normal/images/Normal-5170.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5170.png,Normal/masks/Normal-5170.png,True,False,299,299,L,130.49,358e17ec41ed25be4fb30c94c56a2987
Normal-5171.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5171.png,Normal/images/Normal-5171.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5171.png,Normal/masks/Normal-5171.png,True,False,299,299,L,119.76,8c735f54f57b66f6234bd097bcb23652
Normal-5172.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5172.png,Normal/images/Normal-5172.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5172.png,Normal/masks/Normal-5172.png,True,False,299,299,L,141.43,8eae5b5e76b5ac1fe5aa4908820ee200
Normal-5173.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5173.png,Normal/images/Normal-5173.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5173.png,Normal/masks/Normal-5173.png,True,False,299,299,L,113.39,b64c629ac7fe4696d0bd722e85d73259
Normal-5174.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5174.png,Normal/images/Normal-5174.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5174.png,Normal/masks/Normal-5174.png,True,False,299,299,L,114.22,f14b4943e77fa80d8513ebb92a6d3778
Normal-5175.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5175.png,Normal/images/Normal-5175.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5175.png,Normal/masks/Normal-5175.png,True,False,299,299,L,131.71,6ed1679957ee3de3d6ab2ee0c5d68e66
Normal-5176.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5176.png,Normal/images/Normal-5176.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5176.png,Normal/masks/Normal-5176.png,True,False,299,299,L,111.04,45653a76ff90a8f812b674aac75d804a
Normal-5177.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5177.png,Normal/images/Normal-5177.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5177.png,Normal/masks/Normal-5177.png,True,False,299,299,L,141.05,fc2c3dd3e4a716c8402713bc870d3744
Normal-5178.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5178.png,Normal/images/Normal-5178.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5178.png,Normal/masks/Normal-5178.png,True,False,299,299,L,140.95,243878a09038338be13707182b7b721a
Normal-5179.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5179.png,Normal/images/Normal-5179.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5179.png,Normal/masks/Normal-5179.png,True,False,299,299,L,164.02,6c1171089bfd93ce36383f5e9eb56491
Normal-518.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-518.png,Normal/images/Normal-518.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-518.png,Normal/masks/Normal-518.png,True,False,299,299,L,122.05,17b381d80c41fc7644900cefa77f7558
Normal-5180.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5180.png,Normal/images/Normal-5180.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5180.png,Normal/masks/Normal-5180.png,True,False,299,299,L,108.39,94abb1a31de27d0309d13a9ffc78bf73
Normal-5181.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5181.png,Normal/images/Normal-5181.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5181.png,Normal/masks/Normal-5181.png,True,False,299,299,L,107.87,1eaedeb662861f6858fe42387a5f2292
Normal-5182.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5182.png,Normal/images/Normal-5182.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5182.png,Normal/masks/Normal-5182.png,True,False,299,299,L,168.75,ee64b4440f8b2117e52ceefd70e1120a
Normal-5183.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5183.png,Normal/images/Normal-5183.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5183.png,Normal/masks/Normal-5183.png,True,False,299,299,L,180.83,89d78a2343f493ef62db0f2fdc8d308d
Normal-5184.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5184.png,Normal/images/Normal-5184.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5184.png,Normal/masks/Normal-5184.png,True,False,299,299,L,143.75,16b3d625d5d3f42ad5127cfb81f1cb2d
Normal-5185.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5185.png,Normal/images/Normal-5185.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5185.png,Normal/masks/Normal-5185.png,True,False,299,299,L,155.84,06eff84bfdce2ebf68d9d20f116519fc
Normal-5186.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5186.png,Normal/images/Normal-5186.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5186.png,Normal/masks/Normal-5186.png,True,False,299,299,L,142.33,f38bd08d79dd6f4166680cc60e07207e
Normal-5187.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5187.png,Normal/images/Normal-5187.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5187.png,Normal/masks/Normal-5187.png,True,False,299,299,L,121.13,9709e6d8987b498b04eab1500f4b0263
Normal-5188.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5188.png,Normal/images/Normal-5188.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5188.png,Normal/masks/Normal-5188.png,True,False,299,299,L,151.62,c34958ce3edd9047db7ab0fe36f22a61
Normal-5189.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5189.png,Normal/images/Normal-5189.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5189.png,Normal/masks/Normal-5189.png,True,False,299,299,L,147.94,ee42c84ff8d8d3bf22aed7b11d6b6756
Normal-519.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-519.png,Normal/images/Normal-519.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-519.png,Normal/masks/Normal-519.png,True,False,299,299,L,113.87,2f82c448aec38afcd9f4b76fb3aa9161
Normal-5190.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5190.png,Normal/images/Normal-5190.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5190.png,Normal/masks/Normal-5190.png,True,False,299,299,L,119.27,198566cf14a98953edab913e1d2c52ef
Normal-5191.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5191.png,Normal/images/Normal-5191.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5191.png,Normal/masks/Normal-5191.png,True,False,299,299,L,123.74,8b06e6f263980b71e1f339f885e22d02
Normal-5192.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5192.png,Normal/images/Normal-5192.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5192.png,Normal/masks/Normal-5192.png,True,False,299,299,L,121.89,ec479a1ef4a51c0e07e26376f174fd20
Normal-5193.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5193.png,Normal/images/Normal-5193.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5193.png,Normal/masks/Normal-5193.png,True,False,299,299,L,126.84,0a1a9fd569591eb3afb8ae8c86d2ebe7
Normal-5194.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5194.png,Normal/images/Normal-5194.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5194.png,Normal/masks/Normal-5194.png,True,False,299,299,L,120.75,4026c05be11530d20445b2a72ff6d728
Normal-5195.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5195.png,Normal/images/Normal-5195.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5195.png,Normal/masks/Normal-5195.png,True,False,299,299,L,108.12,a16407dab540b76420cad2a66e9ef9af
Normal-5196.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5196.png,Normal/images/Normal-5196.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5196.png,Normal/masks/Normal-5196.png,True,False,299,299,L,185.09,1fc238bf820b4b37418852fac1711178
Normal-5197.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5197.png,Normal/images/Normal-5197.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5197.png,Normal/masks/Normal-5197.png,True,False,299,299,L,110.14,c97137f767283de86ad367e80ae39fb9
Normal-5198.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5198.png,Normal/images/Normal-5198.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5198.png,Normal/masks/Normal-5198.png,True,False,299,299,L,101.29,fd542d6d4c83e3f8f12d63f6780019cf
Normal-5199.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5199.png,Normal/images/Normal-5199.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5199.png,Normal/masks/Normal-5199.png,True,False,299,299,L,112.24,3f51a0a8da5fa7c0b7a8747205ba0615
Normal-52.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-52.png,Normal/images/Normal-52.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-52.png,Normal/masks/Normal-52.png,True,False,299,299,L,110.93,d86e5550e53ee999132cd1d41ae08425
Normal-520.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-520.png,Normal/images/Normal-520.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-520.png,Normal/masks/Normal-520.png,True,False,299,299,L,108.68,d06571953a4ac4b0adbb7c823fdf32ee
Normal-5200.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5200.png,Normal/images/Normal-5200.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5200.png,Normal/masks/Normal-5200.png,True,False,299,299,L,158.65,710848501254f87dc0fbdc966cbbf709
Normal-5201.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5201.png,Normal/images/Normal-5201.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5201.png,Normal/masks/Normal-5201.png,True,False,299,299,L,113.42,ffabd13f382edeb59163e7181820415e
Normal-5202.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5202.png,Normal/images/Normal-5202.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5202.png,Normal/masks/Normal-5202.png,True,False,299,299,L,120.88,3f91f82d8d7b3eaa24e8667266dce691
Normal-5203.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5203.png,Normal/images/Normal-5203.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5203.png,Normal/masks/Normal-5203.png,True,False,299,299,L,115.21,f9a43b78e481914f661c43278821b945
Normal-5204.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5204.png,Normal/images/Normal-5204.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5204.png,Normal/masks/Normal-5204.png,True,False,299,299,L,69.54,27a35e28fc70347a73d01ee330d8f5ed
Normal-5205.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5205.png,Normal/images/Normal-5205.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5205.png,Normal/masks/Normal-5205.png,True,False,299,299,L,145.84,bbd13f62694abb09f7f1866c8f47a941
Normal-5206.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5206.png,Normal/images/Normal-5206.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5206.png,Normal/masks/Normal-5206.png,True,False,299,299,L,114.34,258ae10dad6f8cd1d40d0aca0edd86ad
Normal-5207.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5207.png,Normal/images/Normal-5207.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5207.png,Normal/masks/Normal-5207.png,True,False,299,299,L,150.18,dd306e0065d109a124875a8300a148be
Normal-5208.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5208.png,Normal/images/Normal-5208.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5208.png,Normal/masks/Normal-5208.png,True,False,299,299,L,142.77,2ada65ce59f8d9fa68b0c74638d6d632
Normal-5209.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5209.png,Normal/images/Normal-5209.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5209.png,Normal/masks/Normal-5209.png,True,False,299,299,L,94.06,2ca9210c8ca3002f283004aace1e1e99
Normal-521.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-521.png,Normal/images/Normal-521.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-521.png,Normal/masks/Normal-521.png,True,False,299,299,L,133.53,da0066557264e9b854d9ea6057eb7daa
Normal-5210.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5210.png,Normal/images/Normal-5210.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5210.png,Normal/masks/Normal-5210.png,True,False,299,299,L,120.98,5424d89218968078d45b14e9fef31852
Normal-5211.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5211.png,Normal/images/Normal-5211.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5211.png,Normal/masks/Normal-5211.png,True,False,299,299,L,144.55,83e4d0377be802b451d8f3a59dc607c0
Normal-5212.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5212.png,Normal/images/Normal-5212.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5212.png,Normal/masks/Normal-5212.png,True,False,299,299,L,122.61,8b09fa6825f38595d4c70a466a15ad34
Normal-5213.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5213.png,Normal/images/Normal-5213.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5213.png,Normal/masks/Normal-5213.png,True,False,299,299,L,133.77,3788ac7af8f81e6fac9c028f318aeb73
Normal-5214.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5214.png,Normal/images/Normal-5214.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5214.png,Normal/masks/Normal-5214.png,True,False,299,299,L,128.92,f9f808266bd06b085e7272374477de1b
Normal-5215.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5215.png,Normal/images/Normal-5215.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5215.png,Normal/masks/Normal-5215.png,True,False,299,299,L,120.4,c2430f1098d42e5a96a6328f955f4dbd
Normal-5216.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5216.png,Normal/images/Normal-5216.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5216.png,Normal/masks/Normal-5216.png,True,False,299,299,L,114.74,5986a7a0c2dbc87373b5f782ffd639b1
Normal-5217.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5217.png,Normal/images/Normal-5217.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5217.png,Normal/masks/Normal-5217.png,True,False,299,299,L,110.63,52a5eef412e9234b070b429f8b7808ec
Normal-5218.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5218.png,Normal/images/Normal-5218.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5218.png,Normal/masks/Normal-5218.png,True,False,299,299,L,163.16,f73f8e2bff42ffc651d1c1e275b905a4
Normal-5219.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5219.png,Normal/images/Normal-5219.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5219.png,Normal/masks/Normal-5219.png,True,False,299,299,L,169.11,accad4bb8d2233272bd44133bbbdd9e1
Normal-522.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-522.png,Normal/images/Normal-522.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-522.png,Normal/masks/Normal-522.png,True,False,299,299,L,114.6,80f88b361c3e21e336100d069b8abb7a
Normal-5220.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5220.png,Normal/images/Normal-5220.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5220.png,Normal/masks/Normal-5220.png,True,False,299,299,L,88.22,17c9fc36fb7d7cf8ee3029aa9bf2ecd5
Normal-5221.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5221.png,Normal/images/Normal-5221.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5221.png,Normal/masks/Normal-5221.png,True,False,299,299,L,107.18,fcfbead43dee28c613d3a46469f67d5f
Normal-5222.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5222.png,Normal/images/Normal-5222.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5222.png,Normal/masks/Normal-5222.png,True,False,299,299,L,119.7,9b5f30ca412a4a5473651114dbf6936c
Normal-5223.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5223.png,Normal/images/Normal-5223.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5223.png,Normal/masks/Normal-5223.png,True,False,299,299,L,144.18,33379dd698a930a5ea1b0a87a72b4ece
Normal-5224.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5224.png,Normal/images/Normal-5224.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5224.png,Normal/masks/Normal-5224.png,True,False,299,299,L,138.72,943b83b501571f06dea84b784174a481
Normal-5225.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5225.png,Normal/images/Normal-5225.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5225.png,Normal/masks/Normal-5225.png,True,False,299,299,L,100.48,8f37e40aab526235114c7c1dbc13fb17
Normal-5226.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5226.png,Normal/images/Normal-5226.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5226.png,Normal/masks/Normal-5226.png,True,False,299,299,L,153.51,d3aee3b7315ba549edba9bc9541608e9
Normal-5227.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5227.png,Normal/images/Normal-5227.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5227.png,Normal/masks/Normal-5227.png,True,False,299,299,L,165.59,a8bf9685e0c9c49a4f0982125112c0c0
Normal-5228.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5228.png,Normal/images/Normal-5228.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5228.png,Normal/masks/Normal-5228.png,True,False,299,299,L,160.1,f68a787b31917c36b9a5fd4a4e93f422
Normal-5229.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5229.png,Normal/images/Normal-5229.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5229.png,Normal/masks/Normal-5229.png,True,False,299,299,L,161.3,c3a8a1b2d5c236af12ceb7fd1c325ee8
Normal-523.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-523.png,Normal/images/Normal-523.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-523.png,Normal/masks/Normal-523.png,True,False,299,299,L,112.78,63c2bcf100fd7b896e725925a490a314
Normal-5230.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5230.png,Normal/images/Normal-5230.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5230.png,Normal/masks/Normal-5230.png,True,False,299,299,L,172.62,70112dc116982300df196fa4fea9747c
Normal-5231.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5231.png,Normal/images/Normal-5231.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5231.png,Normal/masks/Normal-5231.png,True,False,299,299,L,82.98,28cac16864225df678ef4ac5654a29b2
Normal-5232.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5232.png,Normal/images/Normal-5232.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5232.png,Normal/masks/Normal-5232.png,True,False,299,299,L,121.86,f3f66294a18f67b42abbc65d26f7de0b
Normal-5233.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5233.png,Normal/images/Normal-5233.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5233.png,Normal/masks/Normal-5233.png,True,False,299,299,L,117.5,1474ed4547959d6e8ce319419da9e789
Normal-5234.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5234.png,Normal/images/Normal-5234.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5234.png,Normal/masks/Normal-5234.png,True,False,299,299,L,147.45,75a0975f351a18d9234dab14b5f9db25
Normal-5235.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5235.png,Normal/images/Normal-5235.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5235.png,Normal/masks/Normal-5235.png,True,False,299,299,L,140.25,2367768e28e8b4e58c046ff256d78bd6
Normal-5236.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5236.png,Normal/images/Normal-5236.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5236.png,Normal/masks/Normal-5236.png,True,False,299,299,L,118.87,aaa6a789f0e5728ce4fd41256c287c80
Normal-5237.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5237.png,Normal/images/Normal-5237.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5237.png,Normal/masks/Normal-5237.png,True,False,299,299,L,118.04,287cb89e2e13ce5ca1bb4399d5409214
Normal-5238.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5238.png,Normal/images/Normal-5238.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5238.png,Normal/masks/Normal-5238.png,True,False,299,299,L,184.1,7a6cd58b0022a57bc9bd73a0a1a322ac
Normal-5239.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5239.png,Normal/images/Normal-5239.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5239.png,Normal/masks/Normal-5239.png,True,False,299,299,L,117.04,d1d870dc2a456e8cf82943a385bbd1b6
Normal-524.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-524.png,Normal/images/Normal-524.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-524.png,Normal/masks/Normal-524.png,True,False,299,299,L,122.16,8c7153bebc651c8efd36f4b57b239812
Normal-5240.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5240.png,Normal/images/Normal-5240.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5240.png,Normal/masks/Normal-5240.png,True,False,299,299,L,145.78,6812999dfdbe22c6e84c237f3f0dcbae
Normal-5241.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5241.png,Normal/images/Normal-5241.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5241.png,Normal/masks/Normal-5241.png,True,False,299,299,L,103.13,87f4afb9b32096ce66659afcfcb428f4
Normal-5242.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5242.png,Normal/images/Normal-5242.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5242.png,Normal/masks/Normal-5242.png,True,False,299,299,L,104.8,1affeb31c4a2443d121ce16667216a42
Normal-5243.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5243.png,Normal/images/Normal-5243.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5243.png,Normal/masks/Normal-5243.png,True,False,299,299,L,105.35,f17169211dd1f6145c84ced9ef19cfc6
Normal-5244.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5244.png,Normal/images/Normal-5244.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5244.png,Normal/masks/Normal-5244.png,True,False,299,299,L,150.59,0cbf22df0017678642b2dd59b31a0429
Normal-5245.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5245.png,Normal/images/Normal-5245.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5245.png,Normal/masks/Normal-5245.png,True,False,299,299,L,117.49,b9bfa7fcc2cd718288a456d595ad0de4
Normal-5246.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5246.png,Normal/images/Normal-5246.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5246.png,Normal/masks/Normal-5246.png,True,False,299,299,L,138.26,3a09d4cd3d5fa494bff5d413a71514e0
Normal-5247.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5247.png,Normal/images/Normal-5247.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5247.png,Normal/masks/Normal-5247.png,True,False,299,299,L,111.41,0e2a3492bac596906f04d147e4063f47
Normal-5248.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5248.png,Normal/images/Normal-5248.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5248.png,Normal/masks/Normal-5248.png,True,False,299,299,L,114.4,e0b92d5689fdfcacda9c01d44d6f9cc6
Normal-5249.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5249.png,Normal/images/Normal-5249.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5249.png,Normal/masks/Normal-5249.png,True,False,299,299,L,116.08,afbfc75ae3c349c9806809e1d435455e
Normal-525.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-525.png,Normal/images/Normal-525.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-525.png,Normal/masks/Normal-525.png,True,False,299,299,L,144.78,cf4f115f69bffd134eb641b38fe7b126
Normal-5250.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5250.png,Normal/images/Normal-5250.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5250.png,Normal/masks/Normal-5250.png,True,False,299,299,L,99.02,f9242f1d5a49c73f3ab6beb434534e7f
Normal-5251.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5251.png,Normal/images/Normal-5251.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5251.png,Normal/masks/Normal-5251.png,True,False,299,299,L,142.46,04134c5d643491427909d0071aafbba9
Normal-5252.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5252.png,Normal/images/Normal-5252.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5252.png,Normal/masks/Normal-5252.png,True,False,299,299,L,171.47,892c210a7905ed066e7a2af86b3aca77
Normal-5253.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5253.png,Normal/images/Normal-5253.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5253.png,Normal/masks/Normal-5253.png,True,False,299,299,L,91.64,775f16c2be375de18b7373b3da1a2693
Normal-5254.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5254.png,Normal/images/Normal-5254.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5254.png,Normal/masks/Normal-5254.png,True,False,299,299,L,102.23,ee4d5c16fcc5d542612c853a19ae9369
Normal-5255.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5255.png,Normal/images/Normal-5255.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5255.png,Normal/masks/Normal-5255.png,True,False,299,299,L,110.81,d05f80615649428b0f434f96f4f3c298
Normal-5256.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5256.png,Normal/images/Normal-5256.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5256.png,Normal/masks/Normal-5256.png,True,False,299,299,L,153.63,dad07997c8fa906ad00b36f6682250fb
Normal-5257.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5257.png,Normal/images/Normal-5257.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5257.png,Normal/masks/Normal-5257.png,True,False,299,299,L,138.93,61ba96d5e3b8c784f03b987b8a5a9238
Normal-5258.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5258.png,Normal/images/Normal-5258.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5258.png,Normal/masks/Normal-5258.png,True,False,299,299,L,134.04,b26149ec823cbe5a263614b05ef3f9de
Normal-5259.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5259.png,Normal/images/Normal-5259.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5259.png,Normal/masks/Normal-5259.png,True,False,299,299,L,159.7,b3473cca738251792b923d532a381d85
Normal-526.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-526.png,Normal/images/Normal-526.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-526.png,Normal/masks/Normal-526.png,True,False,299,299,L,131.54,68b7ff49d1b91113c31246c2ab0a6f03
Normal-5260.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5260.png,Normal/images/Normal-5260.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5260.png,Normal/masks/Normal-5260.png,True,False,299,299,L,117.26,fe50e0be09511da23714a1737fe102bc
Normal-5261.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5261.png,Normal/images/Normal-5261.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5261.png,Normal/masks/Normal-5261.png,True,False,299,299,L,118.17,65b285d05ea3dbd02e743c7a97b9cb6a
Normal-5262.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5262.png,Normal/images/Normal-5262.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5262.png,Normal/masks/Normal-5262.png,True,False,299,299,L,117.39,017e6203692e71a89f79903da37fb010
Normal-5263.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5263.png,Normal/images/Normal-5263.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5263.png,Normal/masks/Normal-5263.png,True,False,299,299,L,138.74,d0911b63fc313905fb2c64678cb88e3b
Normal-5264.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5264.png,Normal/images/Normal-5264.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5264.png,Normal/masks/Normal-5264.png,True,False,299,299,L,118.56,fced0ecdcaae59e2b121d87381dc1b70
Normal-5265.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5265.png,Normal/images/Normal-5265.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5265.png,Normal/masks/Normal-5265.png,True,False,299,299,L,158.72,529018bc704e06b9141c043a44219c4f
Normal-5266.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5266.png,Normal/images/Normal-5266.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5266.png,Normal/masks/Normal-5266.png,True,False,299,299,L,131.82,20fb4bb6f4ec518468ebbe88dcc0b2ef
Normal-5267.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5267.png,Normal/images/Normal-5267.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5267.png,Normal/masks/Normal-5267.png,True,False,299,299,L,119.69,c6bccf8049f8eeb083549dc6268f2695
Normal-5268.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5268.png,Normal/images/Normal-5268.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5268.png,Normal/masks/Normal-5268.png,True,False,299,299,L,167.91,8028210c0ee7adc85b0efdb34cdbe66c
Normal-5269.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5269.png,Normal/images/Normal-5269.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5269.png,Normal/masks/Normal-5269.png,True,False,299,299,L,138.11,61c5ecf16a70fa370ef4548d6af9303f
Normal-527.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-527.png,Normal/images/Normal-527.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-527.png,Normal/masks/Normal-527.png,True,False,299,299,L,116.95,468197dcfee07a31df894da4be52d556
Normal-5270.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5270.png,Normal/images/Normal-5270.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5270.png,Normal/masks/Normal-5270.png,True,False,299,299,L,110.59,d6df422cd2716d7d4b50eac58584ca42
Normal-5271.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5271.png,Normal/images/Normal-5271.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5271.png,Normal/masks/Normal-5271.png,True,False,299,299,L,147.47,d039a2e2545a01225552e2b574e92b50
Normal-5272.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5272.png,Normal/images/Normal-5272.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5272.png,Normal/masks/Normal-5272.png,True,False,299,299,L,148.12,2fadc1e0991212059908cba41d64d9b1
Normal-5273.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5273.png,Normal/images/Normal-5273.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5273.png,Normal/masks/Normal-5273.png,True,False,299,299,L,116.19,7187bb567b400eafd02d2ee235398e4f
Normal-5274.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5274.png,Normal/images/Normal-5274.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5274.png,Normal/masks/Normal-5274.png,True,False,299,299,L,128.53,493881066a51f33c8e484ac7fbe01182
Normal-5275.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5275.png,Normal/images/Normal-5275.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5275.png,Normal/masks/Normal-5275.png,True,False,299,299,L,105.7,a60d1f64d9f75c4a083fb43a535dfd5d
Normal-5276.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5276.png,Normal/images/Normal-5276.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5276.png,Normal/masks/Normal-5276.png,True,False,299,299,L,96.48,da8b777a60cf983b44197880364753f7
Normal-5277.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5277.png,Normal/images/Normal-5277.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5277.png,Normal/masks/Normal-5277.png,True,False,299,299,L,170.34,04b9d96c7047b6cba99958d3fb15c938
Normal-5278.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5278.png,Normal/images/Normal-5278.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5278.png,Normal/masks/Normal-5278.png,True,False,299,299,L,128.1,fd7e2537d2020a8f4157276c0c6522d2
Normal-5279.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5279.png,Normal/images/Normal-5279.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5279.png,Normal/masks/Normal-5279.png,True,False,299,299,L,136.69,078b538c8cfad3a842b72341117e5a2d
Normal-528.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-528.png,Normal/images/Normal-528.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-528.png,Normal/masks/Normal-528.png,True,False,299,299,L,118.1,73ddcd32c6457a8644c625986e3364fc
Normal-5280.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5280.png,Normal/images/Normal-5280.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5280.png,Normal/masks/Normal-5280.png,True,False,299,299,L,115.61,c4bd23e58976e3a6e81c16a8ad6f375f
Normal-5281.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5281.png,Normal/images/Normal-5281.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5281.png,Normal/masks/Normal-5281.png,True,False,299,299,L,97.98,f46ada984b02ac9824aa0654da75cfac
Normal-5282.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5282.png,Normal/images/Normal-5282.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5282.png,Normal/masks/Normal-5282.png,True,False,299,299,L,120.12,b12d15c5ec8ed722bf7ccb7663013dfa
Normal-5283.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5283.png,Normal/images/Normal-5283.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5283.png,Normal/masks/Normal-5283.png,True,False,299,299,L,113.72,1052aca732124cdd8117b9f50575a594
Normal-5284.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5284.png,Normal/images/Normal-5284.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5284.png,Normal/masks/Normal-5284.png,True,False,299,299,L,103.64,5b20a7ba73edcfaf725cf5150dc66e58
Normal-5285.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5285.png,Normal/images/Normal-5285.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5285.png,Normal/masks/Normal-5285.png,True,False,299,299,L,89.66,efee29892bcad5dd4dda0595ae20c7e6
Normal-5286.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5286.png,Normal/images/Normal-5286.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5286.png,Normal/masks/Normal-5286.png,True,False,299,299,L,170.52,d8b04782622f5a8fd2a0f8cea0ab505e
Normal-5287.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5287.png,Normal/images/Normal-5287.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5287.png,Normal/masks/Normal-5287.png,True,False,299,299,L,126.61,556a9acf540ac3a2ea8d4a3a62d84fec
Normal-5288.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5288.png,Normal/images/Normal-5288.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5288.png,Normal/masks/Normal-5288.png,True,False,299,299,L,155.96,1b9cbec2c9752bba53bd63bd27c1aa96
Normal-5289.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5289.png,Normal/images/Normal-5289.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5289.png,Normal/masks/Normal-5289.png,True,False,299,299,L,116.98,3ca863c7ccabdedc6d4a840033404c9c
Normal-529.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-529.png,Normal/images/Normal-529.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-529.png,Normal/masks/Normal-529.png,True,False,299,299,L,123.18,a31673234a497223da8e578c7fca0b0d
Normal-5290.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5290.png,Normal/images/Normal-5290.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5290.png,Normal/masks/Normal-5290.png,True,False,299,299,L,157.4,ac80daae2baccb7fa53613e748da2372
Normal-5291.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5291.png,Normal/images/Normal-5291.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5291.png,Normal/masks/Normal-5291.png,True,False,299,299,L,93.07,b49c24661eb334782128ea523534d57e
Normal-5292.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5292.png,Normal/images/Normal-5292.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5292.png,Normal/masks/Normal-5292.png,True,False,299,299,L,100.16,2eab9ad96dacc04834850fe2e8b73f63
Normal-5293.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5293.png,Normal/images/Normal-5293.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5293.png,Normal/masks/Normal-5293.png,True,False,299,299,L,108.63,47d0ff4370aa24c2ffce5d70b3bc6d0d
Normal-5294.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5294.png,Normal/images/Normal-5294.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5294.png,Normal/masks/Normal-5294.png,True,False,299,299,L,179.03,0bfee71fc8184664befdf70a4186e74f
Normal-5295.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5295.png,Normal/images/Normal-5295.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5295.png,Normal/masks/Normal-5295.png,True,False,299,299,L,120.33,949073e8127a8ce497d29d52b6c41b9a
Normal-5296.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5296.png,Normal/images/Normal-5296.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5296.png,Normal/masks/Normal-5296.png,True,False,299,299,L,146.5,b43a1a0c21d2dfa64bfa423be595e99d
Normal-5297.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5297.png,Normal/images/Normal-5297.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5297.png,Normal/masks/Normal-5297.png,True,False,299,299,L,98.83,aa232b46b08cf26da24c2110cd536eb6
Normal-5298.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5298.png,Normal/images/Normal-5298.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5298.png,Normal/masks/Normal-5298.png,True,False,299,299,L,105.07,38907468c7728f6b4ce585205f0016db
Normal-5299.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5299.png,Normal/images/Normal-5299.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5299.png,Normal/masks/Normal-5299.png,True,False,299,299,L,160.06,1a950bc381b3b81d97a36d5b6e828e65
Normal-53.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-53.png,Normal/images/Normal-53.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-53.png,Normal/masks/Normal-53.png,True,False,299,299,L,138.83,4b9102f34e085194e3ef576d7d683e13
Normal-530.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-530.png,Normal/images/Normal-530.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-530.png,Normal/masks/Normal-530.png,True,False,299,299,L,129.21,2774ff1eec55c97fc3feab227fb81e17
Normal-5300.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5300.png,Normal/images/Normal-5300.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5300.png,Normal/masks/Normal-5300.png,True,False,299,299,L,148.33,5d5d0dfc57a692466c6b4df3a7a2216f
Normal-5301.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5301.png,Normal/images/Normal-5301.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5301.png,Normal/masks/Normal-5301.png,True,False,299,299,L,153.4,f2ddf380d476198931521ac6301c6a76
Normal-5302.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5302.png,Normal/images/Normal-5302.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5302.png,Normal/masks/Normal-5302.png,True,False,299,299,L,122.84,dacb64a5d22c38f37baecdcca9c75e0a
Normal-5303.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5303.png,Normal/images/Normal-5303.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5303.png,Normal/masks/Normal-5303.png,True,False,299,299,L,140.26,953d0695932d08300260e90601a32c7f
Normal-5304.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5304.png,Normal/images/Normal-5304.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5304.png,Normal/masks/Normal-5304.png,True,False,299,299,L,136.9,7e8220a91b63e1001f4588e6477e168e
Normal-5305.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5305.png,Normal/images/Normal-5305.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5305.png,Normal/masks/Normal-5305.png,True,False,299,299,L,154.77,e6a813ff9d8829f06d600d261bad6e59
Normal-5306.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5306.png,Normal/images/Normal-5306.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5306.png,Normal/masks/Normal-5306.png,True,False,299,299,L,135.06,13c25a98b8ca9f6160cb18ee4c5283db
Normal-5307.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5307.png,Normal/images/Normal-5307.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5307.png,Normal/masks/Normal-5307.png,True,False,299,299,L,144.49,a54ac52eae6fa7da6f69f8d5a34458bd
Normal-5308.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5308.png,Normal/images/Normal-5308.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5308.png,Normal/masks/Normal-5308.png,True,False,299,299,L,126.13,2e1dd095357680d6af14b4ff72f90eb1
Normal-5309.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5309.png,Normal/images/Normal-5309.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5309.png,Normal/masks/Normal-5309.png,True,False,299,299,L,128.75,5a9fb3f51add7cd10042bf99970eabb9
Normal-531.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-531.png,Normal/images/Normal-531.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-531.png,Normal/masks/Normal-531.png,True,False,299,299,L,137.04,66a8525245e110c3a609e8a9cf7d7b4c
Normal-5310.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5310.png,Normal/images/Normal-5310.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5310.png,Normal/masks/Normal-5310.png,True,False,299,299,L,120.82,bdaafb884caeb67504c5f2050eb4c581
Normal-5311.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5311.png,Normal/images/Normal-5311.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5311.png,Normal/masks/Normal-5311.png,True,False,299,299,L,120.35,a9e14b8ec507ef7035b155d523fe0879
Normal-5312.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5312.png,Normal/images/Normal-5312.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5312.png,Normal/masks/Normal-5312.png,True,False,299,299,L,136.13,d3ad26d802d23771aacc1b966f0886a4
Normal-5313.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5313.png,Normal/images/Normal-5313.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5313.png,Normal/masks/Normal-5313.png,True,False,299,299,L,129.34,e95554d7fa340035e15db3612417f363
Normal-5314.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5314.png,Normal/images/Normal-5314.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5314.png,Normal/masks/Normal-5314.png,True,False,299,299,L,156.88,74aa59c7dc2d35d2ec704c1dfe477790
Normal-5315.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5315.png,Normal/images/Normal-5315.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5315.png,Normal/masks/Normal-5315.png,True,False,299,299,L,117.35,679cad0c64643b6442260c40cad68c4d
Normal-5316.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5316.png,Normal/images/Normal-5316.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5316.png,Normal/masks/Normal-5316.png,True,False,299,299,L,100.99,ed96acb279b4607502f720b8978cc638
Normal-5317.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5317.png,Normal/images/Normal-5317.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5317.png,Normal/masks/Normal-5317.png,True,False,299,299,L,127.34,8018da72d2d08a2dfad801348f129d57
Normal-5318.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5318.png,Normal/images/Normal-5318.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5318.png,Normal/masks/Normal-5318.png,True,False,299,299,L,143.17,5483dda4356b740d132c2960ce1ac248
Normal-5319.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5319.png,Normal/images/Normal-5319.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5319.png,Normal/masks/Normal-5319.png,True,False,299,299,L,124.02,9c8ab3cc4aadb725e30263e4da48172c
Normal-532.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-532.png,Normal/images/Normal-532.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-532.png,Normal/masks/Normal-532.png,True,False,299,299,L,129.16,c2d36bf53ebda7fadc462a72f95b8970
Normal-5320.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5320.png,Normal/images/Normal-5320.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5320.png,Normal/masks/Normal-5320.png,True,False,299,299,L,94.49,005863938b2402f86dd06e6b46cec410
Normal-5321.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5321.png,Normal/images/Normal-5321.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5321.png,Normal/masks/Normal-5321.png,True,False,299,299,L,106.57,91614cdee9442116a84e4b780fd2496a
Normal-5322.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5322.png,Normal/images/Normal-5322.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5322.png,Normal/masks/Normal-5322.png,True,False,299,299,L,154.83,313d3cb96a9a148f8932a6d1731c2001
Normal-5323.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5323.png,Normal/images/Normal-5323.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5323.png,Normal/masks/Normal-5323.png,True,False,299,299,L,133.37,27f91b44f6d5a3751a5a771e28b905b3
Normal-5324.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5324.png,Normal/images/Normal-5324.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5324.png,Normal/masks/Normal-5324.png,True,False,299,299,L,159.86,f8f4fa9c208f9d66843a7e51ba0a209a
Normal-5325.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5325.png,Normal/images/Normal-5325.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5325.png,Normal/masks/Normal-5325.png,True,False,299,299,L,114.0,f065b896bd0f2c3cf6eef7c2bc0f6bc9
Normal-5326.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5326.png,Normal/images/Normal-5326.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5326.png,Normal/masks/Normal-5326.png,True,False,299,299,L,157.56,367258879a0c7ab5ad0bad75ad3de7b4
Normal-5327.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5327.png,Normal/images/Normal-5327.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5327.png,Normal/masks/Normal-5327.png,True,False,299,299,L,106.77,e98d86b67862f62adeb1c43094f11ebb
Normal-5328.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5328.png,Normal/images/Normal-5328.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5328.png,Normal/masks/Normal-5328.png,True,False,299,299,L,113.96,8550f6225555cb61203a5e47bb711c4b
Normal-5329.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5329.png,Normal/images/Normal-5329.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5329.png,Normal/masks/Normal-5329.png,True,False,299,299,L,108.2,3a9e35870af945c9aa5439729807de48
Normal-533.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-533.png,Normal/images/Normal-533.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-533.png,Normal/masks/Normal-533.png,True,False,299,299,L,124.05,f7eba21abcf8616867d0819bde2805e5
Normal-5330.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5330.png,Normal/images/Normal-5330.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5330.png,Normal/masks/Normal-5330.png,True,False,299,299,L,135.16,388c169bb0ecbcbb57a13a8c7921ee09
Normal-5331.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5331.png,Normal/images/Normal-5331.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5331.png,Normal/masks/Normal-5331.png,True,False,299,299,L,112.69,0bc38130d381e49dd0e4ba7d03ccceb7
Normal-5332.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5332.png,Normal/images/Normal-5332.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5332.png,Normal/masks/Normal-5332.png,True,False,299,299,L,118.55,ecfb214381f09f75a018cef03a5b0c58
Normal-5333.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5333.png,Normal/images/Normal-5333.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5333.png,Normal/masks/Normal-5333.png,True,False,299,299,L,112.94,b3aa0a1cbb978e7af27ffb855e7d9811
Normal-5334.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5334.png,Normal/images/Normal-5334.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5334.png,Normal/masks/Normal-5334.png,True,False,299,299,L,179.5,a3e89bdbe3f26077e7a487eed83ed3c1
Normal-5335.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5335.png,Normal/images/Normal-5335.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5335.png,Normal/masks/Normal-5335.png,True,False,299,299,L,107.15,422441df71bc14a581dbba599cdd98ca
Normal-5336.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5336.png,Normal/images/Normal-5336.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5336.png,Normal/masks/Normal-5336.png,True,False,299,299,L,125.59,a1925b8350d57e5004423436b82a40fe
Normal-5337.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5337.png,Normal/images/Normal-5337.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5337.png,Normal/masks/Normal-5337.png,True,False,299,299,L,148.38,e7cf91f958f6da25861c9c12a1557cb5
Normal-5338.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5338.png,Normal/images/Normal-5338.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5338.png,Normal/masks/Normal-5338.png,True,False,299,299,L,132.43,1d92178cd440e6635e765f292edaf7a8
Normal-5339.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5339.png,Normal/images/Normal-5339.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5339.png,Normal/masks/Normal-5339.png,True,False,299,299,L,101.73,48d738f5c4dc77c5d0d96f898bbfa631
Normal-534.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-534.png,Normal/images/Normal-534.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-534.png,Normal/masks/Normal-534.png,True,False,299,299,L,124.85,1c51a613f4b2d6f09ed377809a58b73e
Normal-5340.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5340.png,Normal/images/Normal-5340.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5340.png,Normal/masks/Normal-5340.png,True,False,299,299,L,131.71,b2553b9809bb575e8e68f7d97d872ec2
Normal-5341.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5341.png,Normal/images/Normal-5341.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5341.png,Normal/masks/Normal-5341.png,True,False,299,299,L,137.34,79d9ea5db9d673b9ae39228ffc56eda7
Normal-5342.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5342.png,Normal/images/Normal-5342.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5342.png,Normal/masks/Normal-5342.png,True,False,299,299,L,128.8,d3960dd5987e39dfbd1be2c970dd3749
Normal-5343.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5343.png,Normal/images/Normal-5343.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5343.png,Normal/masks/Normal-5343.png,True,False,299,299,L,150.81,4331627905af8b30e64367032928f4de
Normal-5344.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5344.png,Normal/images/Normal-5344.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5344.png,Normal/masks/Normal-5344.png,True,False,299,299,L,105.8,b6977c98de5605bc0aa5de7b60d7442e
Normal-5345.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5345.png,Normal/images/Normal-5345.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5345.png,Normal/masks/Normal-5345.png,True,False,299,299,L,111.63,b731d313f2d75e6190e7c4450145eba7
Normal-5346.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5346.png,Normal/images/Normal-5346.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5346.png,Normal/masks/Normal-5346.png,True,False,299,299,L,165.72,fef9853fd35e87ed16e2004078acdc97
Normal-5347.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5347.png,Normal/images/Normal-5347.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5347.png,Normal/masks/Normal-5347.png,True,False,299,299,L,138.65,9f35e9962128cfa11aae2d4414696d4d
Normal-5348.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5348.png,Normal/images/Normal-5348.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5348.png,Normal/masks/Normal-5348.png,True,False,299,299,L,167.22,8ff54c4959fd4cb3d445f28af45c5814
Normal-5349.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5349.png,Normal/images/Normal-5349.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5349.png,Normal/masks/Normal-5349.png,True,False,299,299,L,144.08,5ffa2f8d6cd313d5e5c04c16fb4795c8
Normal-535.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-535.png,Normal/images/Normal-535.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-535.png,Normal/masks/Normal-535.png,True,False,299,299,L,120.64,60c2b904182c794af2adbe847ff84d8c
Normal-5350.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5350.png,Normal/images/Normal-5350.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5350.png,Normal/masks/Normal-5350.png,True,False,299,299,L,126.44,9447fa01c227bffaff9fef5838dbe4c0
Normal-5351.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5351.png,Normal/images/Normal-5351.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5351.png,Normal/masks/Normal-5351.png,True,False,299,299,L,116.22,f607d771a5a4bdc1ad4df0917a761a54
Normal-5352.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5352.png,Normal/images/Normal-5352.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5352.png,Normal/masks/Normal-5352.png,True,False,299,299,L,122.06,39eb77945858fb5704996d775afff57e
Normal-5353.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5353.png,Normal/images/Normal-5353.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5353.png,Normal/masks/Normal-5353.png,True,False,299,299,L,99.98,33f4f43ca412f38ebe088f0616932896
Normal-5354.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5354.png,Normal/images/Normal-5354.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5354.png,Normal/masks/Normal-5354.png,True,False,299,299,L,133.43,e3e92b5a7b53eac54478d3df3d336e16
Normal-5355.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5355.png,Normal/images/Normal-5355.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5355.png,Normal/masks/Normal-5355.png,True,False,299,299,L,174.54,425ce52e67fe80ee4839698f9109d7ad
Normal-5356.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5356.png,Normal/images/Normal-5356.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5356.png,Normal/masks/Normal-5356.png,True,False,299,299,L,169.53,7291ad2eb6c3c50a3de255a0b6968c9c
Normal-5357.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5357.png,Normal/images/Normal-5357.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5357.png,Normal/masks/Normal-5357.png,True,False,299,299,L,160.52,d0883055f18475a037f578f9b9ac8e7e
Normal-5358.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5358.png,Normal/images/Normal-5358.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5358.png,Normal/masks/Normal-5358.png,True,False,299,299,L,146.48,905b22d978764b00b399f01ea21372a6
Normal-5359.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5359.png,Normal/images/Normal-5359.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5359.png,Normal/masks/Normal-5359.png,True,False,299,299,L,139.79,529fe8f9f95c69ff8d3007a93fcaa28d
Normal-536.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-536.png,Normal/images/Normal-536.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-536.png,Normal/masks/Normal-536.png,True,False,299,299,L,142.72,1ffca2bb966cbdc707553eedc5ea5ba7
Normal-5360.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5360.png,Normal/images/Normal-5360.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5360.png,Normal/masks/Normal-5360.png,True,False,299,299,L,114.25,13903aea8d493e1122137d5b4075420e
Normal-5361.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5361.png,Normal/images/Normal-5361.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5361.png,Normal/masks/Normal-5361.png,True,False,299,299,L,97.57,13d706dc1233f8a84089506239efa8a4
Normal-5362.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5362.png,Normal/images/Normal-5362.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5362.png,Normal/masks/Normal-5362.png,True,False,299,299,L,104.95,3a1a77d6a89d866b5b607f9cfb9641cf
Normal-5363.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5363.png,Normal/images/Normal-5363.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5363.png,Normal/masks/Normal-5363.png,True,False,299,299,L,187.44,ed2acd8cd503992daf8504967226271d
Normal-5364.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5364.png,Normal/images/Normal-5364.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5364.png,Normal/masks/Normal-5364.png,True,False,299,299,L,176.22,20ae7fd0b6af8e807be6df8bf20b363e
Normal-5365.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5365.png,Normal/images/Normal-5365.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5365.png,Normal/masks/Normal-5365.png,True,False,299,299,L,168.38,76466d00221c57f262c9026e31d92dc4
Normal-5366.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5366.png,Normal/images/Normal-5366.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5366.png,Normal/masks/Normal-5366.png,True,False,299,299,L,161.09,538e495c43fcf68af1f8aa7b27d4acdd
Normal-5367.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5367.png,Normal/images/Normal-5367.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5367.png,Normal/masks/Normal-5367.png,True,False,299,299,L,169.97,c40d589d7274e493abbfd7a090597dc9
Normal-5368.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5368.png,Normal/images/Normal-5368.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5368.png,Normal/masks/Normal-5368.png,True,False,299,299,L,123.5,b0acf4c45b926dae0919b31284b10faf
Normal-5369.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5369.png,Normal/images/Normal-5369.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5369.png,Normal/masks/Normal-5369.png,True,False,299,299,L,134.3,3ab434cf5d4210e3eeb1a68b2ec56721
Normal-537.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-537.png,Normal/images/Normal-537.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-537.png,Normal/masks/Normal-537.png,True,False,299,299,L,131.18,be00840a5c0a7a38a057d8d572eaa9dc
Normal-5370.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5370.png,Normal/images/Normal-5370.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5370.png,Normal/masks/Normal-5370.png,True,False,299,299,L,126.56,570b33b512e814e9b0449e46fa0480cd
Normal-5371.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5371.png,Normal/images/Normal-5371.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5371.png,Normal/masks/Normal-5371.png,True,False,299,299,L,118.1,740c2372d9cc6ef06be5f1e8271559b4
Normal-5372.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5372.png,Normal/images/Normal-5372.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5372.png,Normal/masks/Normal-5372.png,True,False,299,299,L,170.74,e65cfcad2ba716ef6676998193125a6e
Normal-5373.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5373.png,Normal/images/Normal-5373.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5373.png,Normal/masks/Normal-5373.png,True,False,299,299,L,121.89,a75b77c7e763d7e0f5d91a01a631a8f8
Normal-5374.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5374.png,Normal/images/Normal-5374.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5374.png,Normal/masks/Normal-5374.png,True,False,299,299,L,147.33,e123ec1b30aa220a8451ad7f79bbfad0
Normal-5375.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5375.png,Normal/images/Normal-5375.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5375.png,Normal/masks/Normal-5375.png,True,False,299,299,L,127.91,56eccd8b7fad4fe82dfeb82866b7a156
Normal-5376.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5376.png,Normal/images/Normal-5376.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5376.png,Normal/masks/Normal-5376.png,True,False,299,299,L,183.87,852b40427bfadb5440e091087709dd49
Normal-5377.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5377.png,Normal/images/Normal-5377.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5377.png,Normal/masks/Normal-5377.png,True,False,299,299,L,166.52,7fe8161cd0ac41fa18510bf7640faaab
Normal-5378.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5378.png,Normal/images/Normal-5378.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5378.png,Normal/masks/Normal-5378.png,True,False,299,299,L,101.35,ff863cf44a317fc2f9d554f3dbdbeb61
Normal-5379.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5379.png,Normal/images/Normal-5379.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5379.png,Normal/masks/Normal-5379.png,True,False,299,299,L,96.84,d03912e78ec1bb543220d3d1895ab1f3
Normal-538.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-538.png,Normal/images/Normal-538.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-538.png,Normal/masks/Normal-538.png,True,False,299,299,L,129.63,004d885b31c0bb9a292af2c1181d89c9
Normal-5380.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5380.png,Normal/images/Normal-5380.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5380.png,Normal/masks/Normal-5380.png,True,False,299,299,L,162.73,58e9c904d8fa3deea8ad31c41afb34da
Normal-5381.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5381.png,Normal/images/Normal-5381.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5381.png,Normal/masks/Normal-5381.png,True,False,299,299,L,145.17,500c406991b3a5d66415c3272c09c3c4
Normal-5382.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5382.png,Normal/images/Normal-5382.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5382.png,Normal/masks/Normal-5382.png,True,False,299,299,L,141.35,6da9100ebec51de4b69f67c1f171cf3d
Normal-5383.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5383.png,Normal/images/Normal-5383.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5383.png,Normal/masks/Normal-5383.png,True,False,299,299,L,163.35,7d89fc0588e5cbd7014fad62d8651abe
Normal-5384.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5384.png,Normal/images/Normal-5384.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5384.png,Normal/masks/Normal-5384.png,True,False,299,299,L,134.21,c8ac76fbfc49e63b59c934c9da503200
Normal-5385.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5385.png,Normal/images/Normal-5385.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5385.png,Normal/masks/Normal-5385.png,True,False,299,299,L,181.89,5486f17a724b6c04289b3699cc197c39
Normal-5386.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5386.png,Normal/images/Normal-5386.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5386.png,Normal/masks/Normal-5386.png,True,False,299,299,L,148.09,1b3866f4f70835af0573758fdbfe6379
Normal-5387.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5387.png,Normal/images/Normal-5387.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5387.png,Normal/masks/Normal-5387.png,True,False,299,299,L,137.51,dc5bbfc3b43fd790b7f09e77931b30d3
Normal-5388.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5388.png,Normal/images/Normal-5388.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5388.png,Normal/masks/Normal-5388.png,True,False,299,299,L,128.52,3ec1799ed2f0e28f6dfdef7a6da67508
Normal-5389.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5389.png,Normal/images/Normal-5389.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5389.png,Normal/masks/Normal-5389.png,True,False,299,299,L,187.05,6a0dade005e936d2296ba3d3bd1c9aff
Normal-539.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-539.png,Normal/images/Normal-539.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-539.png,Normal/masks/Normal-539.png,True,False,299,299,L,112.46,a8c3ba44cebacd2a3c320052c4f25178
Normal-5390.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5390.png,Normal/images/Normal-5390.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5390.png,Normal/masks/Normal-5390.png,True,False,299,299,L,92.84,0f50b365488d1de20cf2268ea6448a3a
Normal-5391.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5391.png,Normal/images/Normal-5391.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5391.png,Normal/masks/Normal-5391.png,True,False,299,299,L,93.06,14892fb97685212f345f9157a50d94aa
Normal-5392.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5392.png,Normal/images/Normal-5392.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5392.png,Normal/masks/Normal-5392.png,True,False,299,299,L,123.71,92dd47a0844b45b6cc4785840fdf961b
Normal-5393.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5393.png,Normal/images/Normal-5393.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5393.png,Normal/masks/Normal-5393.png,True,False,299,299,L,111.93,fa55325a4792e8e931cb999269bca1ec
Normal-5394.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5394.png,Normal/images/Normal-5394.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5394.png,Normal/masks/Normal-5394.png,True,False,299,299,L,108.26,2cf4a94d749f9de74cbc8a0b4561e86e
Normal-5395.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5395.png,Normal/images/Normal-5395.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5395.png,Normal/masks/Normal-5395.png,True,False,299,299,L,112.68,03477c369e9101b39ad1ab8bcd8f8270
Normal-5396.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5396.png,Normal/images/Normal-5396.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5396.png,Normal/masks/Normal-5396.png,True,False,299,299,L,116.86,44e1ce8d7c0c05b2e534c550f1d9c152
Normal-5397.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5397.png,Normal/images/Normal-5397.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5397.png,Normal/masks/Normal-5397.png,True,False,299,299,L,108.41,e3695a57a6f92c8b4e5cdf449b3e5993
Normal-5398.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5398.png,Normal/images/Normal-5398.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5398.png,Normal/masks/Normal-5398.png,True,False,299,299,L,135.85,6ebae663ec32d8ab699ebbf9f3bc92ec
Normal-5399.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5399.png,Normal/images/Normal-5399.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5399.png,Normal/masks/Normal-5399.png,True,False,299,299,L,111.38,46e417272929b58b0f8297103dd14d77
Normal-54.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-54.png,Normal/images/Normal-54.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-54.png,Normal/masks/Normal-54.png,True,False,299,299,L,146.51,d8c7978c7e0f308934f65d1cfa51f059
Normal-540.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-540.png,Normal/images/Normal-540.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-540.png,Normal/masks/Normal-540.png,True,False,299,299,L,131.38,ebc9c7513ae9fdc353ec959c48f6e75b
Normal-5400.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5400.png,Normal/images/Normal-5400.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5400.png,Normal/masks/Normal-5400.png,True,False,299,299,L,136.19,737adff87d95d3a78c6c999e2de89eb4
Normal-5401.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5401.png,Normal/images/Normal-5401.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5401.png,Normal/masks/Normal-5401.png,True,False,299,299,L,109.74,ccbec9d31253427630f0c7a1c3670d5a
Normal-5402.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5402.png,Normal/images/Normal-5402.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5402.png,Normal/masks/Normal-5402.png,True,False,299,299,L,138.04,7cd5ed472ff58382d2ddcced5871c267
Normal-5403.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5403.png,Normal/images/Normal-5403.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5403.png,Normal/masks/Normal-5403.png,True,False,299,299,L,182.22,11df26cea2d3a50d8a10f6b6c95dafe2
Normal-5404.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5404.png,Normal/images/Normal-5404.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5404.png,Normal/masks/Normal-5404.png,True,False,299,299,L,161.24,fcf1ff979209b8940caea257b6ddb17b
Normal-5405.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5405.png,Normal/images/Normal-5405.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5405.png,Normal/masks/Normal-5405.png,True,False,299,299,L,145.37,c621500deaa0cb54b902f201703800c4
Normal-5406.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5406.png,Normal/images/Normal-5406.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5406.png,Normal/masks/Normal-5406.png,True,False,299,299,L,160.63,02af2c8b5ec92de7109c6c0177e824f3
Normal-5407.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5407.png,Normal/images/Normal-5407.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5407.png,Normal/masks/Normal-5407.png,True,False,299,299,L,158.24,fcdae686901bf155b096bc26da6f2acc
Normal-5408.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5408.png,Normal/images/Normal-5408.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5408.png,Normal/masks/Normal-5408.png,True,False,299,299,L,124.37,aba7de2a31b9c7c375af20b37fe8996e
Normal-5409.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5409.png,Normal/images/Normal-5409.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5409.png,Normal/masks/Normal-5409.png,True,False,299,299,L,148.78,bb935875c8f05a82fe51c2d8a5f026d9
Normal-541.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-541.png,Normal/images/Normal-541.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-541.png,Normal/masks/Normal-541.png,True,False,299,299,L,111.75,749331b1a978e437c504d99a4b59f46d
Normal-5410.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5410.png,Normal/images/Normal-5410.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5410.png,Normal/masks/Normal-5410.png,True,False,299,299,L,141.35,da4bd70ea71c7ff53e73643a35faaf7a
Normal-5411.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5411.png,Normal/images/Normal-5411.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5411.png,Normal/masks/Normal-5411.png,True,False,299,299,L,142.89,9fd6896960871ca2d820ade8420c7adf
Normal-5412.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5412.png,Normal/images/Normal-5412.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5412.png,Normal/masks/Normal-5412.png,True,False,299,299,L,183.96,9c44facc1e320d163eae302851f6498f
Normal-5413.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5413.png,Normal/images/Normal-5413.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5413.png,Normal/masks/Normal-5413.png,True,False,299,299,L,112.37,8de34d76757b36c762ce0ecd3a2fb0f0
Normal-5414.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5414.png,Normal/images/Normal-5414.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5414.png,Normal/masks/Normal-5414.png,True,False,299,299,L,163.57,b0aad3d7f91d1cf960b1cad334eefff8
Normal-5415.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5415.png,Normal/images/Normal-5415.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5415.png,Normal/masks/Normal-5415.png,True,False,299,299,L,102.77,8f6bff189f34651f2219306cdacc7a5d
Normal-5416.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5416.png,Normal/images/Normal-5416.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5416.png,Normal/masks/Normal-5416.png,True,False,299,299,L,123.93,73982cc1e850fad16ad482a1e0ff1c66
Normal-5417.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5417.png,Normal/images/Normal-5417.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5417.png,Normal/masks/Normal-5417.png,True,False,299,299,L,111.51,91883d70cdbb177754bb6734c8d2796e
Normal-5418.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5418.png,Normal/images/Normal-5418.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5418.png,Normal/masks/Normal-5418.png,True,False,299,299,L,133.54,f0927879125c3f7fefb8f3b74bb39157
Normal-5419.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5419.png,Normal/images/Normal-5419.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5419.png,Normal/masks/Normal-5419.png,True,False,299,299,L,118.02,569e45f420fbd6952e79cdd2f2b94618
Normal-542.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-542.png,Normal/images/Normal-542.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-542.png,Normal/masks/Normal-542.png,True,False,299,299,L,127.04,53908d3af6323fedb1e8c27624a34a2d
Normal-5420.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5420.png,Normal/images/Normal-5420.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5420.png,Normal/masks/Normal-5420.png,True,False,299,299,L,167.07,f1d12adac588c44a8a62dfcd34700cb9
Normal-5421.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5421.png,Normal/images/Normal-5421.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5421.png,Normal/masks/Normal-5421.png,True,False,299,299,L,169.65,9004eb2b970a93f00535b3a6e303fe2d
Normal-5422.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5422.png,Normal/images/Normal-5422.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5422.png,Normal/masks/Normal-5422.png,True,False,299,299,L,137.91,71e37127800e4ebb1233922691ba5a4d
Normal-5423.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5423.png,Normal/images/Normal-5423.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5423.png,Normal/masks/Normal-5423.png,True,False,299,299,L,130.76,04fcfe9d6d86f9b66c602a5db39e390e
Normal-5424.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5424.png,Normal/images/Normal-5424.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5424.png,Normal/masks/Normal-5424.png,True,False,299,299,L,128.57,32e5a58e5a24ac4c24a0041eb3bfbf5a
Normal-5425.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5425.png,Normal/images/Normal-5425.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5425.png,Normal/masks/Normal-5425.png,True,False,299,299,L,158.99,d52b7e68e8f4cf5ec4c921212da98173
Normal-5426.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5426.png,Normal/images/Normal-5426.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5426.png,Normal/masks/Normal-5426.png,True,False,299,299,L,117.1,33800ea5043a091098910e648b5be19f
Normal-5427.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5427.png,Normal/images/Normal-5427.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5427.png,Normal/masks/Normal-5427.png,True,False,299,299,L,98.88,6879bc8043f92e3d1c1c8b5113cca855
Normal-5428.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5428.png,Normal/images/Normal-5428.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5428.png,Normal/masks/Normal-5428.png,True,False,299,299,L,110.5,5081f78cd8df8cdd7dade7c259c8efdc
Normal-5429.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5429.png,Normal/images/Normal-5429.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5429.png,Normal/masks/Normal-5429.png,True,False,299,299,L,128.64,aec83c4ed4378630b97cc2a8abe6e872
Normal-543.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-543.png,Normal/images/Normal-543.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-543.png,Normal/masks/Normal-543.png,True,False,299,299,L,132.39,541a4f656d50d3b03d4a61fcd7579324
Normal-5430.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5430.png,Normal/images/Normal-5430.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5430.png,Normal/masks/Normal-5430.png,True,False,299,299,L,158.69,1326c2ed63de63d41ad3297ca796e936
Normal-5431.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5431.png,Normal/images/Normal-5431.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5431.png,Normal/masks/Normal-5431.png,True,False,299,299,L,118.47,bfa40c33ccbdf76478d8adc52cb46bdd
Normal-5432.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5432.png,Normal/images/Normal-5432.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5432.png,Normal/masks/Normal-5432.png,True,False,299,299,L,118.91,387f13fb9ed7fed538f7867f12864eed
Normal-5433.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5433.png,Normal/images/Normal-5433.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5433.png,Normal/masks/Normal-5433.png,True,False,299,299,L,105.75,4702d2fb22b31f2ffbfead0b08c44d2d
Normal-5434.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5434.png,Normal/images/Normal-5434.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5434.png,Normal/masks/Normal-5434.png,True,False,299,299,L,111.35,6ae7d4b4b1a7166598b024705c8f8c03
Normal-5435.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5435.png,Normal/images/Normal-5435.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5435.png,Normal/masks/Normal-5435.png,True,False,299,299,L,112.02,07f7b73b7602630fb63b981c58da4141
Normal-5436.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5436.png,Normal/images/Normal-5436.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5436.png,Normal/masks/Normal-5436.png,True,False,299,299,L,151.8,8d649db7aeb4f661b99f3f9160abccf2
Normal-5437.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5437.png,Normal/images/Normal-5437.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5437.png,Normal/masks/Normal-5437.png,True,False,299,299,L,172.07,f84f5f22d9db8c41a00b60f7392688c5
Normal-5438.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5438.png,Normal/images/Normal-5438.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5438.png,Normal/masks/Normal-5438.png,True,False,299,299,L,137.14,eaabc0202654b344e18ee8a243b801b8
Normal-5439.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5439.png,Normal/images/Normal-5439.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5439.png,Normal/masks/Normal-5439.png,True,False,299,299,L,109.19,c791bc20af5c2eac82976699a604433a
Normal-544.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-544.png,Normal/images/Normal-544.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-544.png,Normal/masks/Normal-544.png,True,False,299,299,L,130.12,7d7654fc49a647ef85a17bf3cac4320a
Normal-5440.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5440.png,Normal/images/Normal-5440.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5440.png,Normal/masks/Normal-5440.png,True,False,299,299,L,125.26,b01326e587bd1d6fc59cf112943252f5
Normal-5441.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5441.png,Normal/images/Normal-5441.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5441.png,Normal/masks/Normal-5441.png,True,False,299,299,L,128.4,d7afbe84cca83f00d7b5f3d8c219ce15
Normal-5442.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5442.png,Normal/images/Normal-5442.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5442.png,Normal/masks/Normal-5442.png,True,False,299,299,L,119.19,fc9ab14f0ddcdcb12a026fd7cf30c02c
Normal-5443.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5443.png,Normal/images/Normal-5443.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5443.png,Normal/masks/Normal-5443.png,True,False,299,299,L,180.83,ea212b55148d4333bda11cff1a0626fb
Normal-5444.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5444.png,Normal/images/Normal-5444.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5444.png,Normal/masks/Normal-5444.png,True,False,299,299,L,147.86,66c843969fa718c99902fe8cf499ba36
Normal-5445.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5445.png,Normal/images/Normal-5445.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5445.png,Normal/masks/Normal-5445.png,True,False,299,299,L,110.06,439d86013dffc4ae0c62c0f4853e8faf
Normal-5446.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5446.png,Normal/images/Normal-5446.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5446.png,Normal/masks/Normal-5446.png,True,False,299,299,L,117.79,c563704ff1f2035939dcacddf6de64cf
Normal-5447.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5447.png,Normal/images/Normal-5447.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5447.png,Normal/masks/Normal-5447.png,True,False,299,299,L,148.65,e4f090702cc8b13465a9d177c94963c3
Normal-5448.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5448.png,Normal/images/Normal-5448.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5448.png,Normal/masks/Normal-5448.png,True,False,299,299,L,133.92,5793c86000a5b07f44df9c0a21402f58
Normal-5449.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5449.png,Normal/images/Normal-5449.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5449.png,Normal/masks/Normal-5449.png,True,False,299,299,L,100.27,92fdd457842edb9ab858a78aae8855ad
Normal-545.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-545.png,Normal/images/Normal-545.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-545.png,Normal/masks/Normal-545.png,True,False,299,299,L,131.95,75f847dd15248b4c02380b5f92dbdef2
Normal-5450.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5450.png,Normal/images/Normal-5450.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5450.png,Normal/masks/Normal-5450.png,True,False,299,299,L,136.13,fad622d8be009db53cae229e8480817a
Normal-5451.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5451.png,Normal/images/Normal-5451.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5451.png,Normal/masks/Normal-5451.png,True,False,299,299,L,132.75,36a37f97192777c924d276af07ed7d73
Normal-5452.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5452.png,Normal/images/Normal-5452.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5452.png,Normal/masks/Normal-5452.png,True,False,299,299,L,117.52,f58d5f29ce16dd636bca3933aba4482d
Normal-5453.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5453.png,Normal/images/Normal-5453.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5453.png,Normal/masks/Normal-5453.png,True,False,299,299,L,117.48,99867fa80b4b232ae8d3dc0eb4a21c08
Normal-5454.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5454.png,Normal/images/Normal-5454.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5454.png,Normal/masks/Normal-5454.png,True,False,299,299,L,133.07,1c94ec31a43d8817855a44d539ed1c6b
Normal-5455.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5455.png,Normal/images/Normal-5455.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5455.png,Normal/masks/Normal-5455.png,True,False,299,299,L,168.52,98d48d1b26aa5a14e714fd15e4afea15
Normal-5456.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5456.png,Normal/images/Normal-5456.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5456.png,Normal/masks/Normal-5456.png,True,False,299,299,L,165.12,a8ade6ec33dba75ad2e2582f75cf0bf4
Normal-5457.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5457.png,Normal/images/Normal-5457.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5457.png,Normal/masks/Normal-5457.png,True,False,299,299,L,109.23,55edbdb4ba779568cd78fde86719d757
Normal-5458.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5458.png,Normal/images/Normal-5458.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5458.png,Normal/masks/Normal-5458.png,True,False,299,299,L,128.37,eb4927ddea9c32ba57912698efd4e31c
Normal-5459.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5459.png,Normal/images/Normal-5459.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5459.png,Normal/masks/Normal-5459.png,True,False,299,299,L,147.34,a69322527693ad76d0d9558c86d5e9df
Normal-546.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-546.png,Normal/images/Normal-546.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-546.png,Normal/masks/Normal-546.png,True,False,299,299,L,126.0,faa47eb36b703aa591c4ba8cbe2c8cf6
Normal-5460.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5460.png,Normal/images/Normal-5460.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5460.png,Normal/masks/Normal-5460.png,True,False,299,299,L,179.48,b12ba223006e13e233ab0c7816ac3f90
Normal-5461.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5461.png,Normal/images/Normal-5461.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5461.png,Normal/masks/Normal-5461.png,True,False,299,299,L,128.52,ef69fc81169ec54ed9dd415123d34136
Normal-5462.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5462.png,Normal/images/Normal-5462.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5462.png,Normal/masks/Normal-5462.png,True,False,299,299,L,120.96,7932fd3b819a043993859f5bbf5be740
Normal-5463.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5463.png,Normal/images/Normal-5463.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5463.png,Normal/masks/Normal-5463.png,True,False,299,299,L,171.48,c853776ba2e5a7a5dda734ffd422a405
Normal-5464.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5464.png,Normal/images/Normal-5464.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5464.png,Normal/masks/Normal-5464.png,True,False,299,299,L,124.37,56949f01d0312555d9e18589685031a8
Normal-5465.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5465.png,Normal/images/Normal-5465.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5465.png,Normal/masks/Normal-5465.png,True,False,299,299,L,153.79,802abd0883ec14745d10fa61308e9b75
Normal-5466.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5466.png,Normal/images/Normal-5466.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5466.png,Normal/masks/Normal-5466.png,True,False,299,299,L,171.0,19dd5dd5b296d5464e0d4dabfe448ef7
Normal-5467.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5467.png,Normal/images/Normal-5467.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5467.png,Normal/masks/Normal-5467.png,True,False,299,299,L,112.04,9bfac45fad2a1555b77dc1627716d61b
Normal-5468.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5468.png,Normal/images/Normal-5468.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5468.png,Normal/masks/Normal-5468.png,True,False,299,299,L,162.28,26b1e3ae2d0e168bee7d0de9665d1986
Normal-5469.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5469.png,Normal/images/Normal-5469.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5469.png,Normal/masks/Normal-5469.png,True,False,299,299,L,185.7,dda63d5bf19a76f5b8e555210fae15b8
Normal-547.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-547.png,Normal/images/Normal-547.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-547.png,Normal/masks/Normal-547.png,True,False,299,299,L,129.84,b4da0cb9a52b92ca22db40bfc577eada
Normal-5470.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5470.png,Normal/images/Normal-5470.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5470.png,Normal/masks/Normal-5470.png,True,False,299,299,L,104.05,3e26d8b7557170b6510289f4a4ab01f4
Normal-5471.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5471.png,Normal/images/Normal-5471.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5471.png,Normal/masks/Normal-5471.png,True,False,299,299,L,103.41,b677f21c7313a40419f38d9e3e752bb4
Normal-5472.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5472.png,Normal/images/Normal-5472.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5472.png,Normal/masks/Normal-5472.png,True,False,299,299,L,147.67,e19d249768249bde9a5c0c8374137e4f
Normal-5473.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5473.png,Normal/images/Normal-5473.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5473.png,Normal/masks/Normal-5473.png,True,False,299,299,L,128.26,78d4a18d1523fbe658a683e2e401256e
Normal-5474.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5474.png,Normal/images/Normal-5474.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5474.png,Normal/masks/Normal-5474.png,True,False,299,299,L,135.91,c52e6ba8daccfea37b22a1e93a8be12c
Normal-5475.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5475.png,Normal/images/Normal-5475.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5475.png,Normal/masks/Normal-5475.png,True,False,299,299,L,113.4,708bff08feb9b2279193cbc4cbcaac23
Normal-5476.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5476.png,Normal/images/Normal-5476.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5476.png,Normal/masks/Normal-5476.png,True,False,299,299,L,134.33,178f086760f9070aacc0bb6624a547bf
Normal-5477.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5477.png,Normal/images/Normal-5477.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5477.png,Normal/masks/Normal-5477.png,True,False,299,299,L,136.05,e2d3a224e930ec2c7a93de0e126884d4
Normal-5478.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5478.png,Normal/images/Normal-5478.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5478.png,Normal/masks/Normal-5478.png,True,False,299,299,L,154.41,7df62340bf70701a5f30c1c752172ae7
Normal-5479.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5479.png,Normal/images/Normal-5479.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5479.png,Normal/masks/Normal-5479.png,True,False,299,299,L,141.02,54134b792feeba135b54ccbf68a3d9c0
Normal-548.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-548.png,Normal/images/Normal-548.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-548.png,Normal/masks/Normal-548.png,True,False,299,299,L,116.17,85c7887d8940609391c7dba0dc799374
Normal-5480.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5480.png,Normal/images/Normal-5480.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5480.png,Normal/masks/Normal-5480.png,True,False,299,299,L,178.75,50912fcecbc5a625582224bc9be44116
Normal-5481.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5481.png,Normal/images/Normal-5481.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5481.png,Normal/masks/Normal-5481.png,True,False,299,299,L,157.52,05105931cebb31e742cfa61431b95333
Normal-5482.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5482.png,Normal/images/Normal-5482.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5482.png,Normal/masks/Normal-5482.png,True,False,299,299,L,124.82,0c96ed15a3643dd1984d05a202c45da2
Normal-5483.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5483.png,Normal/images/Normal-5483.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5483.png,Normal/masks/Normal-5483.png,True,False,299,299,L,107.93,af378db17eae47539bd37255fc9b4134
Normal-5484.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5484.png,Normal/images/Normal-5484.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5484.png,Normal/masks/Normal-5484.png,True,False,299,299,L,110.81,30fb3645c77d36374c5f295d8106b80f
Normal-5485.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5485.png,Normal/images/Normal-5485.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5485.png,Normal/masks/Normal-5485.png,True,False,299,299,L,156.17,029e53ab21d6452d7552557da85f1d74
Normal-5486.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5486.png,Normal/images/Normal-5486.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5486.png,Normal/masks/Normal-5486.png,True,False,299,299,L,152.95,2be71126abfa57dc9f5a693e8f377ef5
Normal-5487.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5487.png,Normal/images/Normal-5487.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5487.png,Normal/masks/Normal-5487.png,True,False,299,299,L,132.05,d7721fd613080bde052e29476a860f74
Normal-5488.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5488.png,Normal/images/Normal-5488.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5488.png,Normal/masks/Normal-5488.png,True,False,299,299,L,126.85,9b1719984921fc23b28327b300633aff
Normal-5489.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5489.png,Normal/images/Normal-5489.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5489.png,Normal/masks/Normal-5489.png,True,False,299,299,L,176.29,d06355a630d0acc51cfad1510d5f3292
Normal-549.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-549.png,Normal/images/Normal-549.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-549.png,Normal/masks/Normal-549.png,True,False,299,299,L,139.91,accd60b0f3cd9c5bc355bcb3bce51310
Normal-5490.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5490.png,Normal/images/Normal-5490.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5490.png,Normal/masks/Normal-5490.png,True,False,299,299,L,156.06,03b7722fbe2eed0e20bd3e6a7a4ac1a8
Normal-5491.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5491.png,Normal/images/Normal-5491.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5491.png,Normal/masks/Normal-5491.png,True,False,299,299,L,123.75,68ed259902194dd5c8e4f2a842878495
Normal-5492.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5492.png,Normal/images/Normal-5492.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5492.png,Normal/masks/Normal-5492.png,True,False,299,299,L,138.57,c458b7e5034faff714b8f957630e6fb6
Normal-5493.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5493.png,Normal/images/Normal-5493.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5493.png,Normal/masks/Normal-5493.png,True,False,299,299,L,138.28,a129025735bfd3d3f51b64513d7a1fc6
Normal-5494.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5494.png,Normal/images/Normal-5494.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5494.png,Normal/masks/Normal-5494.png,True,False,299,299,L,129.97,707c1e489fe07a5e31cb1d5c0c51f20d
Normal-5495.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5495.png,Normal/images/Normal-5495.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5495.png,Normal/masks/Normal-5495.png,True,False,299,299,L,120.88,50dbff5e6ca1ea26c4c7e521df15a890
Normal-5496.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5496.png,Normal/images/Normal-5496.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5496.png,Normal/masks/Normal-5496.png,True,False,299,299,L,132.56,17cd94fa0cc54a2fab473b5c1b79a592
Normal-5497.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5497.png,Normal/images/Normal-5497.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5497.png,Normal/masks/Normal-5497.png,True,False,299,299,L,119.63,2b0a8b6d7130bf85265542423908016e
Normal-5498.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5498.png,Normal/images/Normal-5498.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5498.png,Normal/masks/Normal-5498.png,True,False,299,299,L,133.24,9411570b0257cdf38e29df983b10182a
Normal-5499.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5499.png,Normal/images/Normal-5499.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5499.png,Normal/masks/Normal-5499.png,True,False,299,299,L,152.53,b9198417e6be5d31af42492ac9079454
Normal-55.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-55.png,Normal/images/Normal-55.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-55.png,Normal/masks/Normal-55.png,True,False,299,299,L,146.36,75c0443f327b7f239d52a7b38573a212
Normal-550.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-550.png,Normal/images/Normal-550.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-550.png,Normal/masks/Normal-550.png,True,False,299,299,L,130.16,5651e8e95d6991408b6d577ce6e351a2
Normal-5500.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5500.png,Normal/images/Normal-5500.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5500.png,Normal/masks/Normal-5500.png,True,False,299,299,L,108.79,413451dded1bf74884ff8199a8adb574
Normal-5501.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5501.png,Normal/images/Normal-5501.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5501.png,Normal/masks/Normal-5501.png,True,False,299,299,L,172.21,19ea52c53efaa01b907f65aafbaa1b09
Normal-5502.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5502.png,Normal/images/Normal-5502.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5502.png,Normal/masks/Normal-5502.png,True,False,299,299,L,112.13,f5677a6633e59fab1cd3214dce2567dd
Normal-5503.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5503.png,Normal/images/Normal-5503.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5503.png,Normal/masks/Normal-5503.png,True,False,299,299,L,141.23,cec8d56c5f5af51bafe8e0dca22c93ff
Normal-5504.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5504.png,Normal/images/Normal-5504.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5504.png,Normal/masks/Normal-5504.png,True,False,299,299,L,114.19,82f7573026b3820faa66eb93bced4863
Normal-5505.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5505.png,Normal/images/Normal-5505.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5505.png,Normal/masks/Normal-5505.png,True,False,299,299,L,129.14,94fb4e36f6a315c7bd87caba8ad25cee
Normal-5506.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5506.png,Normal/images/Normal-5506.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5506.png,Normal/masks/Normal-5506.png,True,False,299,299,L,113.34,7f0ddd13e5baf2f68b5c63307a1292a3
Normal-5507.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5507.png,Normal/images/Normal-5507.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5507.png,Normal/masks/Normal-5507.png,True,False,299,299,L,164.07,e78309c9555d29749d5f0ca4268253c3
Normal-5508.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5508.png,Normal/images/Normal-5508.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5508.png,Normal/masks/Normal-5508.png,True,False,299,299,L,179.48,b0968b8231fba37c2217c94ae240d08c
Normal-5509.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5509.png,Normal/images/Normal-5509.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5509.png,Normal/masks/Normal-5509.png,True,False,299,299,L,153.75,94e8c99d10f49ccccee92caaac69cbfe
Normal-551.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-551.png,Normal/images/Normal-551.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-551.png,Normal/masks/Normal-551.png,True,False,299,299,L,102.33,52a2e9cc9a48dc028c4bb3d6fc35f03e
Normal-5510.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5510.png,Normal/images/Normal-5510.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5510.png,Normal/masks/Normal-5510.png,True,False,299,299,L,151.14,f77206acdbed75ffb456b5e3ca899c4d
Normal-5511.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5511.png,Normal/images/Normal-5511.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5511.png,Normal/masks/Normal-5511.png,True,False,299,299,L,163.42,2fe12c030838ca93f45d334eac0ddf50
Normal-5512.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5512.png,Normal/images/Normal-5512.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5512.png,Normal/masks/Normal-5512.png,True,False,299,299,L,129.38,21971cc9fa74b39129b4d455f70659bd
Normal-5513.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5513.png,Normal/images/Normal-5513.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5513.png,Normal/masks/Normal-5513.png,True,False,299,299,L,148.97,8ebe8dbb1881ceb0b73f128dd7e92bfc
Normal-5514.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5514.png,Normal/images/Normal-5514.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5514.png,Normal/masks/Normal-5514.png,True,False,299,299,L,162.26,f29eb93f400228ef448c60600626385f
Normal-5515.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5515.png,Normal/images/Normal-5515.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5515.png,Normal/masks/Normal-5515.png,True,False,299,299,L,128.08,6fe5ed8e016fc9161e9fd4f01a61d296
Normal-5516.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5516.png,Normal/images/Normal-5516.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5516.png,Normal/masks/Normal-5516.png,True,False,299,299,L,128.64,b86534cbb58769d0b7d6e972f2333ded
Normal-5517.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5517.png,Normal/images/Normal-5517.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5517.png,Normal/masks/Normal-5517.png,True,False,299,299,L,164.88,4157e48fcb26cf3068bd452209ac6622
Normal-5518.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5518.png,Normal/images/Normal-5518.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5518.png,Normal/masks/Normal-5518.png,True,False,299,299,L,160.0,bb9a952e865b1b84b2a59ada754e052f
Normal-5519.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5519.png,Normal/images/Normal-5519.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5519.png,Normal/masks/Normal-5519.png,True,False,299,299,L,88.35,3b05696dcc782f691bdb44fbafa59ca9
Normal-552.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-552.png,Normal/images/Normal-552.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-552.png,Normal/masks/Normal-552.png,True,False,299,299,L,139.17,515c6f41164813dacb609d806c0682c7
Normal-5520.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5520.png,Normal/images/Normal-5520.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5520.png,Normal/masks/Normal-5520.png,True,False,299,299,L,117.28,e1cbac1ff0807415bdbd5e9056858132
Normal-5521.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5521.png,Normal/images/Normal-5521.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5521.png,Normal/masks/Normal-5521.png,True,False,299,299,L,164.24,ce55931470c1e0be74a59458a2f88c21
Normal-5522.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5522.png,Normal/images/Normal-5522.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5522.png,Normal/masks/Normal-5522.png,True,False,299,299,L,160.57,23e72f228ee268fc966ada06e4efc403
Normal-5523.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5523.png,Normal/images/Normal-5523.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5523.png,Normal/masks/Normal-5523.png,True,False,299,299,L,119.14,6b0345234fd4fb9a5de83da8d8077ea0
Normal-5524.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5524.png,Normal/images/Normal-5524.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5524.png,Normal/masks/Normal-5524.png,True,False,299,299,L,107.07,e4848aa16b6d6d61d6bb6a845391a00c
Normal-5525.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5525.png,Normal/images/Normal-5525.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5525.png,Normal/masks/Normal-5525.png,True,False,299,299,L,123.13,8202472e280b5019e22eb1e0a372315b
Normal-5526.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5526.png,Normal/images/Normal-5526.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5526.png,Normal/masks/Normal-5526.png,True,False,299,299,L,123.92,bd53bcff5f6fdc80aef63bf18f202500
Normal-5527.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5527.png,Normal/images/Normal-5527.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5527.png,Normal/masks/Normal-5527.png,True,False,299,299,L,118.05,66b86ac86f1edddd2218f6fd46b914d9
Normal-5528.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5528.png,Normal/images/Normal-5528.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5528.png,Normal/masks/Normal-5528.png,True,False,299,299,L,140.91,ca5ab09ee213c3243fe98486595fc9da
Normal-5529.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5529.png,Normal/images/Normal-5529.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5529.png,Normal/masks/Normal-5529.png,True,False,299,299,L,128.89,dc913e5bc51bfc9b39e01aff7ed653f6
Normal-553.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-553.png,Normal/images/Normal-553.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-553.png,Normal/masks/Normal-553.png,True,False,299,299,L,117.69,d7d6ebf51665ed1ca202c03cba8a6d72
Normal-5530.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5530.png,Normal/images/Normal-5530.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5530.png,Normal/masks/Normal-5530.png,True,False,299,299,L,131.84,07f416204916fdcc08859a618d738254
Normal-5531.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5531.png,Normal/images/Normal-5531.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5531.png,Normal/masks/Normal-5531.png,True,False,299,299,L,112.44,dd85fe2ebeaf47fa1421b068bbfc98cb
Normal-5532.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5532.png,Normal/images/Normal-5532.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5532.png,Normal/masks/Normal-5532.png,True,False,299,299,L,102.79,9197e8c22b71183349cf6efb6f245b1c
Normal-5533.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5533.png,Normal/images/Normal-5533.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5533.png,Normal/masks/Normal-5533.png,True,False,299,299,L,105.23,8ec7c15d92c9c9d2317819b2eb2434cc
Normal-5534.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5534.png,Normal/images/Normal-5534.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5534.png,Normal/masks/Normal-5534.png,True,False,299,299,L,170.41,66b316464f3c0dea0136b9d29f8887c6
Normal-5535.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5535.png,Normal/images/Normal-5535.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5535.png,Normal/masks/Normal-5535.png,True,False,299,299,L,130.57,332fdaf52a057df499a43a6c237edba9
Normal-5536.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5536.png,Normal/images/Normal-5536.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5536.png,Normal/masks/Normal-5536.png,True,False,299,299,L,156.92,27f773c8165e599e21458d05e603a862
Normal-5537.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5537.png,Normal/images/Normal-5537.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5537.png,Normal/masks/Normal-5537.png,True,False,299,299,L,137.63,d32832b3cfd1ddaf03783696a80d15cf
Normal-5538.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5538.png,Normal/images/Normal-5538.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5538.png,Normal/masks/Normal-5538.png,True,False,299,299,L,137.13,410a88037696e6c082d63320440653a2
Normal-5539.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5539.png,Normal/images/Normal-5539.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5539.png,Normal/masks/Normal-5539.png,True,False,299,299,L,123.1,2f13b99a950ffb023887eef4bc93d901
Normal-554.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-554.png,Normal/images/Normal-554.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-554.png,Normal/masks/Normal-554.png,True,False,299,299,L,135.19,f5bab22bfc4044e5c5fc1cb62c438e75
Normal-5540.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5540.png,Normal/images/Normal-5540.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5540.png,Normal/masks/Normal-5540.png,True,False,299,299,L,136.52,e6ccdeb1e99b742690efd8c7c6be7db5
Normal-5541.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5541.png,Normal/images/Normal-5541.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5541.png,Normal/masks/Normal-5541.png,True,False,299,299,L,114.87,2fd231187a8fd11e0e7186334d9a7324
Normal-5542.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5542.png,Normal/images/Normal-5542.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5542.png,Normal/masks/Normal-5542.png,True,False,299,299,L,163.98,52cfe7cced9778795ff3092f77a49cd9
Normal-5543.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5543.png,Normal/images/Normal-5543.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5543.png,Normal/masks/Normal-5543.png,True,False,299,299,L,139.82,3a38d0179ccf946ed509a4dac7a0a3db
Normal-5544.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5544.png,Normal/images/Normal-5544.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5544.png,Normal/masks/Normal-5544.png,True,False,299,299,L,116.7,5fc9c152c62c5a8530ce9bdc3c4d7efb
Normal-5545.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5545.png,Normal/images/Normal-5545.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5545.png,Normal/masks/Normal-5545.png,True,False,299,299,L,144.53,25cbdc4bfc3e4f15b3b6c2f5159279ab
Normal-5546.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5546.png,Normal/images/Normal-5546.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5546.png,Normal/masks/Normal-5546.png,True,False,299,299,L,108.71,d19240afef2eb62387f5a94ecc8adb1b
Normal-5547.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5547.png,Normal/images/Normal-5547.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5547.png,Normal/masks/Normal-5547.png,True,False,299,299,L,66.51,d82c505394b630379da49df158f02db3
Normal-5548.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5548.png,Normal/images/Normal-5548.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5548.png,Normal/masks/Normal-5548.png,True,False,299,299,L,96.0,45e7fa2b0e9bf4a7d2cfb448edb3de42
Normal-5549.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5549.png,Normal/images/Normal-5549.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5549.png,Normal/masks/Normal-5549.png,True,False,299,299,L,145.12,7893bdf5e87504a841af9a99dfdf1d8b
Normal-555.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-555.png,Normal/images/Normal-555.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-555.png,Normal/masks/Normal-555.png,True,False,299,299,L,110.3,181ae963e91f7e59b18d1267851c4032
Normal-5550.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5550.png,Normal/images/Normal-5550.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5550.png,Normal/masks/Normal-5550.png,True,False,299,299,L,151.46,041cef2549ac05d3f2d9682e935ac792
Normal-5551.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5551.png,Normal/images/Normal-5551.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5551.png,Normal/masks/Normal-5551.png,True,False,299,299,L,137.63,1b6d8277c332b91fb6f70b9184f943a8
Normal-5552.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5552.png,Normal/images/Normal-5552.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5552.png,Normal/masks/Normal-5552.png,True,False,299,299,L,174.68,5e44437b0de7e6f2c7f3016f130cede8
Normal-5553.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5553.png,Normal/images/Normal-5553.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5553.png,Normal/masks/Normal-5553.png,True,False,299,299,L,178.84,c934209f4e2bdf156b2c9118b350b497
Normal-5554.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5554.png,Normal/images/Normal-5554.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5554.png,Normal/masks/Normal-5554.png,True,False,299,299,L,101.03,2c4a7e71ecf63b8076088da5a531c4ea
Normal-5555.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5555.png,Normal/images/Normal-5555.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5555.png,Normal/masks/Normal-5555.png,True,False,299,299,L,147.79,7207caee6a47579a13697cbc3aaf54c7
Normal-5556.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5556.png,Normal/images/Normal-5556.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5556.png,Normal/masks/Normal-5556.png,True,False,299,299,L,140.82,b1990fe9a2bebb71372e32e14a05ed56
Normal-5557.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5557.png,Normal/images/Normal-5557.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5557.png,Normal/masks/Normal-5557.png,True,False,299,299,L,136.86,4cb8266f188285834e5cfdc2156b1b5e
Normal-5558.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5558.png,Normal/images/Normal-5558.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5558.png,Normal/masks/Normal-5558.png,True,False,299,299,L,167.64,20583373a1712cfa396dcec0e4c92b9c
Normal-5559.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5559.png,Normal/images/Normal-5559.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5559.png,Normal/masks/Normal-5559.png,True,False,299,299,L,109.69,d22eaa80fa0ca8c851f1a17f4556c674
Normal-556.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-556.png,Normal/images/Normal-556.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-556.png,Normal/masks/Normal-556.png,True,False,299,299,L,111.24,8f2cc1ce28abb21bb17142b4004e8db7
Normal-5560.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5560.png,Normal/images/Normal-5560.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5560.png,Normal/masks/Normal-5560.png,True,False,299,299,L,120.98,d9020afb3d4416eefe3a1d62b5a28f94
Normal-5561.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5561.png,Normal/images/Normal-5561.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5561.png,Normal/masks/Normal-5561.png,True,False,299,299,L,146.02,fc1e7293feafec21d2ee5aed92503b2a
Normal-5562.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5562.png,Normal/images/Normal-5562.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5562.png,Normal/masks/Normal-5562.png,True,False,299,299,L,151.33,09934181a35cf25591a4dc4ac186faf1
Normal-5563.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5563.png,Normal/images/Normal-5563.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5563.png,Normal/masks/Normal-5563.png,True,False,299,299,L,122.24,a3653b447c9ce307b5d52bfcff7a6eaf
Normal-5564.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5564.png,Normal/images/Normal-5564.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5564.png,Normal/masks/Normal-5564.png,True,False,299,299,L,130.18,68cefba6462d4def93383f0a8e670aca
Normal-5565.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5565.png,Normal/images/Normal-5565.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5565.png,Normal/masks/Normal-5565.png,True,False,299,299,L,134.13,a9a2c9ed8b49e4c464ffbb45ca0b2608
Normal-5566.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5566.png,Normal/images/Normal-5566.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5566.png,Normal/masks/Normal-5566.png,True,False,299,299,L,152.57,7b4f9aa2c2652f295e2a4250a99b95d9
Normal-5567.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5567.png,Normal/images/Normal-5567.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5567.png,Normal/masks/Normal-5567.png,True,False,299,299,L,126.53,4cf029936f464063ba934bf2a0eeaca4
Normal-5568.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5568.png,Normal/images/Normal-5568.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5568.png,Normal/masks/Normal-5568.png,True,False,299,299,L,139.54,2871f69f22701b540c283002259e040c
Normal-5569.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5569.png,Normal/images/Normal-5569.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5569.png,Normal/masks/Normal-5569.png,True,False,299,299,L,137.4,3656cc5fe7c23db59934fed5bd333aca
Normal-557.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-557.png,Normal/images/Normal-557.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-557.png,Normal/masks/Normal-557.png,True,False,299,299,L,115.43,ecdef0c4a5fadb98f3a5ebc8a967b412
Normal-5570.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5570.png,Normal/images/Normal-5570.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5570.png,Normal/masks/Normal-5570.png,True,False,299,299,L,159.41,3fc4ffb1f444f5f3c92e353f024edcb4
Normal-5571.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5571.png,Normal/images/Normal-5571.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5571.png,Normal/masks/Normal-5571.png,True,False,299,299,L,118.23,f08ac23d60e3c22a15a61613741cba87
Normal-5572.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5572.png,Normal/images/Normal-5572.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5572.png,Normal/masks/Normal-5572.png,True,False,299,299,L,96.12,a1b20e2ff5e6440130e301e7a0a231da
Normal-5573.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5573.png,Normal/images/Normal-5573.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5573.png,Normal/masks/Normal-5573.png,True,False,299,299,L,131.87,6ea634c336924aaba18a9d6c504c4b3f
Normal-5574.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5574.png,Normal/images/Normal-5574.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5574.png,Normal/masks/Normal-5574.png,True,False,299,299,L,99.76,de3138826764d27c82960bffecb9e2f0
Normal-5575.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5575.png,Normal/images/Normal-5575.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5575.png,Normal/masks/Normal-5575.png,True,False,299,299,L,125.8,4fb7be8db6303e7239ac940a105c0021
Normal-5576.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5576.png,Normal/images/Normal-5576.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5576.png,Normal/masks/Normal-5576.png,True,False,299,299,L,142.73,94a88897d4a676be0a2735ac4eeeb722
Normal-5577.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5577.png,Normal/images/Normal-5577.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5577.png,Normal/masks/Normal-5577.png,True,False,299,299,L,106.34,cf0c985ce9ed5dcb5f18ca4cce5252dc
Normal-5578.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5578.png,Normal/images/Normal-5578.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5578.png,Normal/masks/Normal-5578.png,True,False,299,299,L,150.98,123a0b57bd9167d0054e1172f4a89ae7
Normal-5579.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5579.png,Normal/images/Normal-5579.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5579.png,Normal/masks/Normal-5579.png,True,False,299,299,L,129.65,3340b148efbcef7114daa62e2cf2169b
Normal-558.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-558.png,Normal/images/Normal-558.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-558.png,Normal/masks/Normal-558.png,True,False,299,299,L,91.33,8aaf0e551ad682349831ac3a6e3c6b0c
Normal-5580.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5580.png,Normal/images/Normal-5580.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5580.png,Normal/masks/Normal-5580.png,True,False,299,299,L,174.55,5dca6c2ff1047bfd8e4e58b9dba7507f
Normal-5581.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5581.png,Normal/images/Normal-5581.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5581.png,Normal/masks/Normal-5581.png,True,False,299,299,L,162.7,24afa3eb6fcdf6127c029ff593def9e6
Normal-5582.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5582.png,Normal/images/Normal-5582.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5582.png,Normal/masks/Normal-5582.png,True,False,299,299,L,112.57,d9887de603646a846f21ace6496dfb08
Normal-5583.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5583.png,Normal/images/Normal-5583.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5583.png,Normal/masks/Normal-5583.png,True,False,299,299,L,136.82,1691b922bc92e9ecc0864b466ad57b6c
Normal-5584.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5584.png,Normal/images/Normal-5584.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5584.png,Normal/masks/Normal-5584.png,True,False,299,299,L,103.86,d2e07839f85a73513bddc407430a550d
Normal-5585.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5585.png,Normal/images/Normal-5585.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5585.png,Normal/masks/Normal-5585.png,True,False,299,299,L,127.14,78bc6d2b3060a10aae038d8950f1cd3b
Normal-5586.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5586.png,Normal/images/Normal-5586.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5586.png,Normal/masks/Normal-5586.png,True,False,299,299,L,120.59,633233bdae8df7c2ff76076a8a91e0e1
Normal-5587.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5587.png,Normal/images/Normal-5587.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5587.png,Normal/masks/Normal-5587.png,True,False,299,299,L,113.99,d7de30e6f4f8e8a6e5646838752ae45b
Normal-5588.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5588.png,Normal/images/Normal-5588.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5588.png,Normal/masks/Normal-5588.png,True,False,299,299,L,134.64,ddf865087a9d1f330f3204b72357bc0a
Normal-5589.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5589.png,Normal/images/Normal-5589.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5589.png,Normal/masks/Normal-5589.png,True,False,299,299,L,176.96,1f2e66552a100526d39aee6174120d85
Normal-559.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-559.png,Normal/images/Normal-559.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-559.png,Normal/masks/Normal-559.png,True,False,299,299,L,130.99,35226eba10cc9ab648e8d9cd4a26f7a0
Normal-5590.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5590.png,Normal/images/Normal-5590.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5590.png,Normal/masks/Normal-5590.png,True,False,299,299,L,148.86,606cb56529690340633fca197441f2a8
Normal-5591.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5591.png,Normal/images/Normal-5591.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5591.png,Normal/masks/Normal-5591.png,True,False,299,299,L,171.66,b9c8fdc1496b870519112169bbe27349
Normal-5592.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5592.png,Normal/images/Normal-5592.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5592.png,Normal/masks/Normal-5592.png,True,False,299,299,L,128.48,63375f005cdf0ca313b39b3cc6b43ed2
Normal-5593.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5593.png,Normal/images/Normal-5593.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5593.png,Normal/masks/Normal-5593.png,True,False,299,299,L,134.75,1ffa5f88b167c092c19dddfedfa3797a
Normal-5594.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5594.png,Normal/images/Normal-5594.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5594.png,Normal/masks/Normal-5594.png,True,False,299,299,L,163.9,a70f3349c0e643b3a8bb4343ba60a7dc
Normal-5595.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5595.png,Normal/images/Normal-5595.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5595.png,Normal/masks/Normal-5595.png,True,False,299,299,L,120.68,7ff41556683c2a3a7554ca05fc0a600e
Normal-5596.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5596.png,Normal/images/Normal-5596.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5596.png,Normal/masks/Normal-5596.png,True,False,299,299,L,101.37,8d9b21c2f78d5d2195c22248c7f43e70
Normal-5597.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5597.png,Normal/images/Normal-5597.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5597.png,Normal/masks/Normal-5597.png,True,False,299,299,L,105.18,8894b04d554c4af4321d62284fcef82a
Normal-5598.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5598.png,Normal/images/Normal-5598.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5598.png,Normal/masks/Normal-5598.png,True,False,299,299,L,116.35,a729060f1606e7ad28e39880dabffb3b
Normal-5599.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5599.png,Normal/images/Normal-5599.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5599.png,Normal/masks/Normal-5599.png,True,False,299,299,L,86.11,75f9f22756ad60bc2abdc2d19f0f6348
Normal-56.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-56.png,Normal/images/Normal-56.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-56.png,Normal/masks/Normal-56.png,True,False,299,299,L,163.43,62637df4a1cec5c48f03898cabd32076
Normal-560.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-560.png,Normal/images/Normal-560.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-560.png,Normal/masks/Normal-560.png,True,False,299,299,L,146.37,876f3131e2489138d11c167aa05b8457
Normal-5600.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5600.png,Normal/images/Normal-5600.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5600.png,Normal/masks/Normal-5600.png,True,False,299,299,L,105.02,b2010731dd2983db8530cfc8611139e0
Normal-5601.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5601.png,Normal/images/Normal-5601.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5601.png,Normal/masks/Normal-5601.png,True,False,299,299,L,115.05,0c9ec0df64bf813fe3a1cee40d2432bb
Normal-5602.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5602.png,Normal/images/Normal-5602.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5602.png,Normal/masks/Normal-5602.png,True,False,299,299,L,141.8,dd69d24a09603f5fcac3a8e117b6a5ce
Normal-5603.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5603.png,Normal/images/Normal-5603.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5603.png,Normal/masks/Normal-5603.png,True,False,299,299,L,98.44,6e0675588be83d2c2f8015effc047bd4
Normal-5604.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5604.png,Normal/images/Normal-5604.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5604.png,Normal/masks/Normal-5604.png,True,False,299,299,L,115.31,d7b0880aea60c0d0f74f3f76ca4cf751
Normal-5605.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5605.png,Normal/images/Normal-5605.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5605.png,Normal/masks/Normal-5605.png,True,False,299,299,L,104.63,ba9e87863f7d616ef100efea9a6256ff
Normal-5606.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5606.png,Normal/images/Normal-5606.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5606.png,Normal/masks/Normal-5606.png,True,False,299,299,L,161.03,80648aa297956f8b3b919f3a55bcb1d4
Normal-5607.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5607.png,Normal/images/Normal-5607.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5607.png,Normal/masks/Normal-5607.png,True,False,299,299,L,98.1,9a4909706cb0c93663d69b0548707909
Normal-5608.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5608.png,Normal/images/Normal-5608.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5608.png,Normal/masks/Normal-5608.png,True,False,299,299,L,114.43,e042ef9fc73933786742a4ab2d1981cf
Normal-5609.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5609.png,Normal/images/Normal-5609.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5609.png,Normal/masks/Normal-5609.png,True,False,299,299,L,105.55,20667b8f178435d0acafd2a72c4e228e
Normal-561.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-561.png,Normal/images/Normal-561.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-561.png,Normal/masks/Normal-561.png,True,False,299,299,L,120.01,e2bc264f01101958d5af7d5cdab4c19b
Normal-5610.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5610.png,Normal/images/Normal-5610.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5610.png,Normal/masks/Normal-5610.png,True,False,299,299,L,122.3,6dc6ac1967187519ee7f16011486cdcb
Normal-5611.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5611.png,Normal/images/Normal-5611.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5611.png,Normal/masks/Normal-5611.png,True,False,299,299,L,126.21,a8bbf02f1a781dfb4d85c3d4e6c30c4f
Normal-5612.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5612.png,Normal/images/Normal-5612.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5612.png,Normal/masks/Normal-5612.png,True,False,299,299,L,127.32,e8d622befaf741365ecc0de08bfd73fd
Normal-5613.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5613.png,Normal/images/Normal-5613.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5613.png,Normal/masks/Normal-5613.png,True,False,299,299,L,106.64,3ff0e22244a6c5513f18fac8f9dcbfc9
Normal-5614.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5614.png,Normal/images/Normal-5614.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5614.png,Normal/masks/Normal-5614.png,True,False,299,299,L,100.8,e300b0f6c8062ccebd108cef95e382ec
Normal-5615.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5615.png,Normal/images/Normal-5615.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5615.png,Normal/masks/Normal-5615.png,True,False,299,299,L,127.66,340cb679810e1432125138b4b824ce25
Normal-5616.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5616.png,Normal/images/Normal-5616.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5616.png,Normal/masks/Normal-5616.png,True,False,299,299,L,144.19,7927894a80a112853f57c623b95c679b
Normal-5617.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5617.png,Normal/images/Normal-5617.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5617.png,Normal/masks/Normal-5617.png,True,False,299,299,L,166.14,4c44ce57564cd21328f1c38b49e9afac
Normal-5618.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5618.png,Normal/images/Normal-5618.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5618.png,Normal/masks/Normal-5618.png,True,False,299,299,L,130.55,b33ce2808c7da05470734055f722945c
Normal-5619.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5619.png,Normal/images/Normal-5619.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5619.png,Normal/masks/Normal-5619.png,True,False,299,299,L,100.22,00b85b921f79c4224023abfcbac561bb
Normal-562.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-562.png,Normal/images/Normal-562.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-562.png,Normal/masks/Normal-562.png,True,False,299,299,L,129.7,505bb653a8bd4abf39bb12961973e20b
Normal-5620.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5620.png,Normal/images/Normal-5620.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5620.png,Normal/masks/Normal-5620.png,True,False,299,299,L,147.16,fdf4ffbc55aac420a8070ee6549cff0e
Normal-5621.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5621.png,Normal/images/Normal-5621.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5621.png,Normal/masks/Normal-5621.png,True,False,299,299,L,126.96,fdac2ad393c0767793c050613d01c51e
Normal-5622.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5622.png,Normal/images/Normal-5622.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5622.png,Normal/masks/Normal-5622.png,True,False,299,299,L,123.54,912215126df72243a1cb9d20c4906a12
Normal-5623.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5623.png,Normal/images/Normal-5623.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5623.png,Normal/masks/Normal-5623.png,True,False,299,299,L,115.91,fbf76e33fc658b3d73742eeb52245190
Normal-5624.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5624.png,Normal/images/Normal-5624.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5624.png,Normal/masks/Normal-5624.png,True,False,299,299,L,124.66,61a85988da3a9ea1418c8c866e3ca34f
Normal-5625.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5625.png,Normal/images/Normal-5625.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5625.png,Normal/masks/Normal-5625.png,True,False,299,299,L,120.36,1cc5d3ca705736009656a6932ec41c6f
Normal-5626.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5626.png,Normal/images/Normal-5626.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5626.png,Normal/masks/Normal-5626.png,True,False,299,299,L,116.05,0191d308f68819a3a6d93e6b5bf9e7fb
Normal-5627.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5627.png,Normal/images/Normal-5627.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5627.png,Normal/masks/Normal-5627.png,True,False,299,299,L,126.48,ba3f7bddc26b1ce76e2451fe5471f5b7
Normal-5628.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5628.png,Normal/images/Normal-5628.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5628.png,Normal/masks/Normal-5628.png,True,False,299,299,L,121.37,02582b5fd66ab7725038f3a4b4496a83
Normal-5629.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5629.png,Normal/images/Normal-5629.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5629.png,Normal/masks/Normal-5629.png,True,False,299,299,L,149.64,6232b044cd93b376bf9b2bab7b2ea038
Normal-563.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-563.png,Normal/images/Normal-563.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-563.png,Normal/masks/Normal-563.png,True,False,299,299,L,148.15,a53cc84f2695365dbe0ab27267c76a53
Normal-5630.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5630.png,Normal/images/Normal-5630.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5630.png,Normal/masks/Normal-5630.png,True,False,299,299,L,107.38,4eb935ec4970f07953e6c029461c66e8
Normal-5631.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5631.png,Normal/images/Normal-5631.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5631.png,Normal/masks/Normal-5631.png,True,False,299,299,L,113.45,65fb32be72009b83eca44ef7135d6855
Normal-5632.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5632.png,Normal/images/Normal-5632.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5632.png,Normal/masks/Normal-5632.png,True,False,299,299,L,119.38,c7aecca594f2913a3b280580c4638124
Normal-5633.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5633.png,Normal/images/Normal-5633.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5633.png,Normal/masks/Normal-5633.png,True,False,299,299,L,122.97,b6e4afe689670ebe7f9f2d193c544a6d
Normal-5634.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5634.png,Normal/images/Normal-5634.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5634.png,Normal/masks/Normal-5634.png,True,False,299,299,L,113.55,3c58c39c0bea7c6616a1c1c0aa25ee65
Normal-5635.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5635.png,Normal/images/Normal-5635.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5635.png,Normal/masks/Normal-5635.png,True,False,299,299,L,73.9,f75af5b9e2eb5dc2baf4075250843461
Normal-5636.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5636.png,Normal/images/Normal-5636.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5636.png,Normal/masks/Normal-5636.png,True,False,299,299,L,105.62,83582e3b212e57f3477d1f145f81b626
Normal-5637.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5637.png,Normal/images/Normal-5637.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5637.png,Normal/masks/Normal-5637.png,True,False,299,299,L,140.48,b22110fff8c09a0e3ca0ea95ee2d8cd1
Normal-5638.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5638.png,Normal/images/Normal-5638.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5638.png,Normal/masks/Normal-5638.png,True,False,299,299,L,162.56,cd98758642a0d9902a8afd09037006c1
Normal-5639.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5639.png,Normal/images/Normal-5639.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5639.png,Normal/masks/Normal-5639.png,True,False,299,299,L,177.95,4284ace68320b2a2d5334fb776f9cd83
Normal-564.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-564.png,Normal/images/Normal-564.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-564.png,Normal/masks/Normal-564.png,True,False,299,299,L,114.02,b59bb784850bcdeee20de5a438c3ec9c
Normal-5640.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5640.png,Normal/images/Normal-5640.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5640.png,Normal/masks/Normal-5640.png,True,False,299,299,L,165.73,ffeaaa65458c5ebfb64ac0c2cc63850b
Normal-5641.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5641.png,Normal/images/Normal-5641.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5641.png,Normal/masks/Normal-5641.png,True,False,299,299,L,165.13,842d5376897fe156987f4249a4fe9c86
Normal-5642.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5642.png,Normal/images/Normal-5642.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5642.png,Normal/masks/Normal-5642.png,True,False,299,299,L,187.13,46aee08a6891b520a8202aa26f085def
Normal-5643.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5643.png,Normal/images/Normal-5643.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5643.png,Normal/masks/Normal-5643.png,True,False,299,299,L,95.83,23ea947a7955c9480073c3a959c3ac2f
Normal-5644.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5644.png,Normal/images/Normal-5644.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5644.png,Normal/masks/Normal-5644.png,True,False,299,299,L,127.75,a57014d3a2e3ffb438db888123fc8106
Normal-5645.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5645.png,Normal/images/Normal-5645.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5645.png,Normal/masks/Normal-5645.png,True,False,299,299,L,177.39,37db46585347ae311bdb656cf7c86c51
Normal-5646.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5646.png,Normal/images/Normal-5646.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5646.png,Normal/masks/Normal-5646.png,True,False,299,299,L,98.85,74c426c55b494ea574039e2e64facffa
Normal-5647.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5647.png,Normal/images/Normal-5647.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5647.png,Normal/masks/Normal-5647.png,True,False,299,299,L,103.77,2eeeaf25c655d6a53b5f9e8c7e9daa14
Normal-5648.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5648.png,Normal/images/Normal-5648.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5648.png,Normal/masks/Normal-5648.png,True,False,299,299,L,154.02,4e60034940619a82a9e070b21fea7719
Normal-5649.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5649.png,Normal/images/Normal-5649.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5649.png,Normal/masks/Normal-5649.png,True,False,299,299,L,170.13,9d2308d5e95de636ad53e889ea1b8923
Normal-565.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-565.png,Normal/images/Normal-565.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-565.png,Normal/masks/Normal-565.png,True,False,299,299,L,126.86,32b1cfbee1a9b4456a3a843a734888d6
Normal-5650.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5650.png,Normal/images/Normal-5650.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5650.png,Normal/masks/Normal-5650.png,True,False,299,299,L,127.37,005e099b687d8431bfd30eea3fb877cf
Normal-5651.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5651.png,Normal/images/Normal-5651.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5651.png,Normal/masks/Normal-5651.png,True,False,299,299,L,125.91,7c2ec50985d7f6a7f843e825743d3e87
Normal-5652.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5652.png,Normal/images/Normal-5652.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5652.png,Normal/masks/Normal-5652.png,True,False,299,299,L,170.89,cb3320f78c84663ab0f04d67adfcbdfd
Normal-5653.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5653.png,Normal/images/Normal-5653.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5653.png,Normal/masks/Normal-5653.png,True,False,299,299,L,100.8,0e9366dbf1c53a587412a231351dafbb
Normal-5654.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5654.png,Normal/images/Normal-5654.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5654.png,Normal/masks/Normal-5654.png,True,False,299,299,L,128.41,cfc4b198e5dd88b0bdcda7d5bf760d74
Normal-5655.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5655.png,Normal/images/Normal-5655.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5655.png,Normal/masks/Normal-5655.png,True,False,299,299,L,163.38,df194172eb85979803d07de8594615a7
Normal-5656.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5656.png,Normal/images/Normal-5656.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5656.png,Normal/masks/Normal-5656.png,True,False,299,299,L,97.29,47971837d7218f4831b05f87ee84f668
Normal-5657.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5657.png,Normal/images/Normal-5657.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5657.png,Normal/masks/Normal-5657.png,True,False,299,299,L,138.61,f19e63fc69bf6a0c9e8885c1db4c7fab
Normal-5658.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5658.png,Normal/images/Normal-5658.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5658.png,Normal/masks/Normal-5658.png,True,False,299,299,L,122.22,88de024c26ffe63dc7da97e8b843e359
Normal-5659.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5659.png,Normal/images/Normal-5659.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5659.png,Normal/masks/Normal-5659.png,True,False,299,299,L,122.2,2908fef3d8088755330ce576aad23958
Normal-566.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-566.png,Normal/images/Normal-566.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-566.png,Normal/masks/Normal-566.png,True,False,299,299,L,137.52,3e370a80f01ffdeec2cdebb57eb74199
Normal-5660.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5660.png,Normal/images/Normal-5660.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5660.png,Normal/masks/Normal-5660.png,True,False,299,299,L,128.44,26bd7662adbd9850ba901b233fe9c481
Normal-5661.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5661.png,Normal/images/Normal-5661.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5661.png,Normal/masks/Normal-5661.png,True,False,299,299,L,115.7,cd801be1f2312881cc23ec1643889a2a
Normal-5662.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5662.png,Normal/images/Normal-5662.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5662.png,Normal/masks/Normal-5662.png,True,False,299,299,L,153.75,91d10bac2767efa8568c7f6a6363810d
Normal-5663.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5663.png,Normal/images/Normal-5663.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5663.png,Normal/masks/Normal-5663.png,True,False,299,299,L,131.83,0f998081d7558607bd4d72e60459a228
Normal-5664.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5664.png,Normal/images/Normal-5664.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5664.png,Normal/masks/Normal-5664.png,True,False,299,299,L,119.77,72d61024d9e8443a83827e78f91300f8
Normal-5665.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5665.png,Normal/images/Normal-5665.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5665.png,Normal/masks/Normal-5665.png,True,False,299,299,L,108.4,36b8232f3b2eda1abc01689172d6bae8
Normal-5666.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5666.png,Normal/images/Normal-5666.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5666.png,Normal/masks/Normal-5666.png,True,False,299,299,L,172.68,e4d324a5e79ff9427f5b470d51447bd6
Normal-5667.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5667.png,Normal/images/Normal-5667.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5667.png,Normal/masks/Normal-5667.png,True,False,299,299,L,164.57,61f7eb8bf3c5b4442037116b02af7ab1
Normal-5668.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5668.png,Normal/images/Normal-5668.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5668.png,Normal/masks/Normal-5668.png,True,False,299,299,L,120.09,e7bf094b4ffcb478b2f30a2ce569f206
Normal-5669.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5669.png,Normal/images/Normal-5669.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5669.png,Normal/masks/Normal-5669.png,True,False,299,299,L,130.53,6e8451e2601f2f6fab3b0d919319313d
Normal-567.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-567.png,Normal/images/Normal-567.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-567.png,Normal/masks/Normal-567.png,True,False,299,299,L,121.59,6b95a5df39f7f432bdfd0bc2dd51a9e7
Normal-5670.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5670.png,Normal/images/Normal-5670.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5670.png,Normal/masks/Normal-5670.png,True,False,299,299,L,135.99,570c8370092d6c71b760b293a4a43d37
Normal-5671.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5671.png,Normal/images/Normal-5671.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5671.png,Normal/masks/Normal-5671.png,True,False,299,299,L,149.27,a40bec0189953f2623c3c23c14ad75bd
Normal-5672.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5672.png,Normal/images/Normal-5672.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5672.png,Normal/masks/Normal-5672.png,True,False,299,299,L,171.33,7b1508eadd1c400fdadd0eadeca56af4
Normal-5673.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5673.png,Normal/images/Normal-5673.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5673.png,Normal/masks/Normal-5673.png,True,False,299,299,L,123.11,f671112e8e8d1d964cd842e8b95662df
Normal-5674.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5674.png,Normal/images/Normal-5674.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5674.png,Normal/masks/Normal-5674.png,True,False,299,299,L,136.12,61c5a969eb2d2637d361811ef6a1fb55
Normal-5675.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5675.png,Normal/images/Normal-5675.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5675.png,Normal/masks/Normal-5675.png,True,False,299,299,L,160.95,c9756820fc269a29d04fed3d1f3b1256
Normal-5676.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5676.png,Normal/images/Normal-5676.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5676.png,Normal/masks/Normal-5676.png,True,False,299,299,L,158.02,c487c1a8af90f51db8fa42777313af07
Normal-5677.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5677.png,Normal/images/Normal-5677.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5677.png,Normal/masks/Normal-5677.png,True,False,299,299,L,103.01,e86f2e0c1e5715fc244734b20ebba7e9
Normal-5678.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5678.png,Normal/images/Normal-5678.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5678.png,Normal/masks/Normal-5678.png,True,False,299,299,L,126.36,221bc476c57a4fd25e69e56a81006355
Normal-5679.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5679.png,Normal/images/Normal-5679.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5679.png,Normal/masks/Normal-5679.png,True,False,299,299,L,124.98,8bfb7bb126014520ad0b3501706095d9
Normal-568.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-568.png,Normal/images/Normal-568.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-568.png,Normal/masks/Normal-568.png,True,False,299,299,L,123.54,ebdb15810e25fd010e7c9de9caf6b4e3
Normal-5680.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5680.png,Normal/images/Normal-5680.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5680.png,Normal/masks/Normal-5680.png,True,False,299,299,L,87.73,5b701807286be43f3375929f3c211e3c
Normal-5681.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5681.png,Normal/images/Normal-5681.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5681.png,Normal/masks/Normal-5681.png,True,False,299,299,L,117.56,69d99115c1db960a01a64751a9248f71
Normal-5682.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5682.png,Normal/images/Normal-5682.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5682.png,Normal/masks/Normal-5682.png,True,False,299,299,L,132.95,62aeca98db4dba355d53e33ef0525481
Normal-5683.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5683.png,Normal/images/Normal-5683.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5683.png,Normal/masks/Normal-5683.png,True,False,299,299,L,122.36,5a35dae10e22d86dc7dd68435105e4b0
Normal-5684.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5684.png,Normal/images/Normal-5684.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5684.png,Normal/masks/Normal-5684.png,True,False,299,299,L,138.95,2a77d3dfbbd8b8f2cc3f5295d017db0d
Normal-5685.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5685.png,Normal/images/Normal-5685.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5685.png,Normal/masks/Normal-5685.png,True,False,299,299,L,173.97,7176d0981a50bdd5fe861c9f318aa7ec
Normal-5686.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5686.png,Normal/images/Normal-5686.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5686.png,Normal/masks/Normal-5686.png,True,False,299,299,L,151.25,5316e329c72dcd0c73f457971764861f
Normal-5687.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5687.png,Normal/images/Normal-5687.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5687.png,Normal/masks/Normal-5687.png,True,False,299,299,L,150.98,d9d0158ed371534ec0097c05d85bb3c4
Normal-5688.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5688.png,Normal/images/Normal-5688.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5688.png,Normal/masks/Normal-5688.png,True,False,299,299,L,162.66,5c7714a988082e9f421ff77b6ad04a0d
Normal-5689.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5689.png,Normal/images/Normal-5689.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5689.png,Normal/masks/Normal-5689.png,True,False,299,299,L,127.11,487ac1e680373ebfefeafcb73ef2635d
Normal-569.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-569.png,Normal/images/Normal-569.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-569.png,Normal/masks/Normal-569.png,True,False,299,299,L,125.81,5463f450b3eeaa08abf6415ea9ca6bda
Normal-5690.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5690.png,Normal/images/Normal-5690.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5690.png,Normal/masks/Normal-5690.png,True,False,299,299,L,121.24,107bbb04179d8ad4fc8b24b34511f346
Normal-5691.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5691.png,Normal/images/Normal-5691.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5691.png,Normal/masks/Normal-5691.png,True,False,299,299,L,109.31,f1dcdff80043871341af5aa5f49dfed0
Normal-5692.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5692.png,Normal/images/Normal-5692.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5692.png,Normal/masks/Normal-5692.png,True,False,299,299,L,123.91,6122e71c353101df58a69368e96b411f
Normal-5693.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5693.png,Normal/images/Normal-5693.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5693.png,Normal/masks/Normal-5693.png,True,False,299,299,L,133.47,7ae8e3d7dd49234cd42facc54506c4fe
Normal-5694.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5694.png,Normal/images/Normal-5694.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5694.png,Normal/masks/Normal-5694.png,True,False,299,299,L,120.52,5a0e4ecf64e44732f194d69e1a836aa7
Normal-5695.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5695.png,Normal/images/Normal-5695.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5695.png,Normal/masks/Normal-5695.png,True,False,299,299,L,98.57,fb3d81e41eb664b96e5797205806404b
Normal-5696.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5696.png,Normal/images/Normal-5696.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5696.png,Normal/masks/Normal-5696.png,True,False,299,299,L,95.32,0d2cffbbf85dc422cf8ffdafa48d79f2
Normal-5697.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5697.png,Normal/images/Normal-5697.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5697.png,Normal/masks/Normal-5697.png,True,False,299,299,L,144.92,b7c24fff3951be77676c180356959694
Normal-5698.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5698.png,Normal/images/Normal-5698.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5698.png,Normal/masks/Normal-5698.png,True,False,299,299,L,145.62,597d66b2c67baacdf16fe3db47e4bd78
Normal-5699.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5699.png,Normal/images/Normal-5699.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5699.png,Normal/masks/Normal-5699.png,True,False,299,299,L,109.0,8b1ff9d66afa320fb813090a6768b5e9
Normal-57.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-57.png,Normal/images/Normal-57.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-57.png,Normal/masks/Normal-57.png,True,False,299,299,L,157.58,efa6f8a9a7b269a653926cc5a71e384f
Normal-570.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-570.png,Normal/images/Normal-570.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-570.png,Normal/masks/Normal-570.png,True,False,299,299,L,104.39,6cc92b8a0f32fff1403bd23388591725
Normal-5700.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5700.png,Normal/images/Normal-5700.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5700.png,Normal/masks/Normal-5700.png,True,False,299,299,L,129.08,52106d4d236f9544a71dd454b48ca16a
Normal-5701.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5701.png,Normal/images/Normal-5701.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5701.png,Normal/masks/Normal-5701.png,True,False,299,299,L,114.27,d27bad4d34384d6c7a6c8c969004ece2
Normal-5702.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5702.png,Normal/images/Normal-5702.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5702.png,Normal/masks/Normal-5702.png,True,False,299,299,L,103.01,86a4c346ed7eb1a3eef294c8b7e5c0f7
Normal-5703.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5703.png,Normal/images/Normal-5703.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5703.png,Normal/masks/Normal-5703.png,True,False,299,299,L,111.41,8ea5f649c65703bf22c896d6d03bfd15
Normal-5704.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5704.png,Normal/images/Normal-5704.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5704.png,Normal/masks/Normal-5704.png,True,False,299,299,L,116.15,27b9ef9a4872a125e6056ba57b0bfc18
Normal-5705.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5705.png,Normal/images/Normal-5705.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5705.png,Normal/masks/Normal-5705.png,True,False,299,299,L,119.49,05d8652e0f758722f830302f6261ea40
Normal-5706.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5706.png,Normal/images/Normal-5706.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5706.png,Normal/masks/Normal-5706.png,True,False,299,299,L,129.23,436445720acb1a7eaaba82676595e2c9
Normal-5707.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5707.png,Normal/images/Normal-5707.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5707.png,Normal/masks/Normal-5707.png,True,False,299,299,L,139.2,3713f8c2037f0db6baec831697874899
Normal-5708.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5708.png,Normal/images/Normal-5708.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5708.png,Normal/masks/Normal-5708.png,True,False,299,299,L,118.47,05343eaa0ecd4a892be2b04314147ae0
Normal-5709.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5709.png,Normal/images/Normal-5709.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5709.png,Normal/masks/Normal-5709.png,True,False,299,299,L,123.86,868ee7e11edb5e076092ff3974a25afc
Normal-571.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-571.png,Normal/images/Normal-571.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-571.png,Normal/masks/Normal-571.png,True,False,299,299,L,114.51,3bad402a3e87e9c3a8f779c26a62ae36
Normal-5710.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5710.png,Normal/images/Normal-5710.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5710.png,Normal/masks/Normal-5710.png,True,False,299,299,L,111.61,78f690fd9e369ff5792b32bf8fdd18cd
Normal-5711.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5711.png,Normal/images/Normal-5711.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5711.png,Normal/masks/Normal-5711.png,True,False,299,299,L,108.3,f519079ec64d107a54866f6ac7e8fee0
Normal-5712.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5712.png,Normal/images/Normal-5712.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5712.png,Normal/masks/Normal-5712.png,True,False,299,299,L,98.65,bd5c34ab03fffa4cb2b41f56893f3ab5
Normal-5713.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5713.png,Normal/images/Normal-5713.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5713.png,Normal/masks/Normal-5713.png,True,False,299,299,L,120.18,2a9fdeda6668d724f3f90c7ddb1a2404
Normal-5714.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5714.png,Normal/images/Normal-5714.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5714.png,Normal/masks/Normal-5714.png,True,False,299,299,L,134.89,583aea4a1c160ed1890f35995e25a384
Normal-5715.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5715.png,Normal/images/Normal-5715.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5715.png,Normal/masks/Normal-5715.png,True,False,299,299,L,166.18,0619f00b7cb178bbd53d66b7805b2473
Normal-5716.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5716.png,Normal/images/Normal-5716.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5716.png,Normal/masks/Normal-5716.png,True,False,299,299,L,113.68,5a82e60f313af14fb55a67320467c138
Normal-5717.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5717.png,Normal/images/Normal-5717.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5717.png,Normal/masks/Normal-5717.png,True,False,299,299,L,130.0,d463d0dd83e478a6edf373065d2008f2
Normal-5718.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5718.png,Normal/images/Normal-5718.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5718.png,Normal/masks/Normal-5718.png,True,False,299,299,L,121.03,9484f376fe3306faeb244c68584fed19
Normal-5719.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5719.png,Normal/images/Normal-5719.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5719.png,Normal/masks/Normal-5719.png,True,False,299,299,L,118.16,cde41ecb875852644b984a511d72890f
Normal-572.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-572.png,Normal/images/Normal-572.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-572.png,Normal/masks/Normal-572.png,True,False,299,299,L,126.52,a1839fcc90b4999abb713e3814c11b56
Normal-5720.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5720.png,Normal/images/Normal-5720.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5720.png,Normal/masks/Normal-5720.png,True,False,299,299,L,156.83,c9a8d0753672156c183a421b14b642b1
Normal-5721.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5721.png,Normal/images/Normal-5721.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5721.png,Normal/masks/Normal-5721.png,True,False,299,299,L,96.17,a4b4b1604e9602807370745ca4cd9d02
Normal-5722.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5722.png,Normal/images/Normal-5722.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5722.png,Normal/masks/Normal-5722.png,True,False,299,299,L,110.2,00c089ce46fc1b5a0726e7dc825af765
Normal-5723.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5723.png,Normal/images/Normal-5723.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5723.png,Normal/masks/Normal-5723.png,True,False,299,299,L,101.95,2497a1f590b6c47a4a31b661419aa62c
Normal-5724.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5724.png,Normal/images/Normal-5724.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5724.png,Normal/masks/Normal-5724.png,True,False,299,299,L,119.53,29d7f3367cca09cb301b97ddbf3a402c
Normal-5725.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5725.png,Normal/images/Normal-5725.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5725.png,Normal/masks/Normal-5725.png,True,False,299,299,L,100.53,58d670d6cca68867dc0ce1712b7c980d
Normal-5726.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5726.png,Normal/images/Normal-5726.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5726.png,Normal/masks/Normal-5726.png,True,False,299,299,L,104.78,e3048025246730837568ca180ef6aba1
Normal-5727.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5727.png,Normal/images/Normal-5727.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5727.png,Normal/masks/Normal-5727.png,True,False,299,299,L,103.79,f45d25792f30ae35c0f418cf0e6ad2ce
Normal-5728.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5728.png,Normal/images/Normal-5728.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5728.png,Normal/masks/Normal-5728.png,True,False,299,299,L,123.73,0012f61a66b99b316e5bda792e1c72ab
Normal-5729.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5729.png,Normal/images/Normal-5729.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5729.png,Normal/masks/Normal-5729.png,True,False,299,299,L,117.03,ba166bc30176950df27e620430ece8f8
Normal-573.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-573.png,Normal/images/Normal-573.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-573.png,Normal/masks/Normal-573.png,True,False,299,299,L,119.08,24a8ffaf481c142d7d6713bce87056d6
Normal-5730.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5730.png,Normal/images/Normal-5730.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5730.png,Normal/masks/Normal-5730.png,True,False,299,299,L,118.39,fd789016be5433797449484639d5e70c
Normal-5731.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5731.png,Normal/images/Normal-5731.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5731.png,Normal/masks/Normal-5731.png,True,False,299,299,L,106.38,f1d0c9ccdfed206bf3124a4eed91793c
Normal-5732.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5732.png,Normal/images/Normal-5732.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5732.png,Normal/masks/Normal-5732.png,True,False,299,299,L,137.67,a2e6294404590897978c3015a8c0493f
Normal-5733.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5733.png,Normal/images/Normal-5733.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5733.png,Normal/masks/Normal-5733.png,True,False,299,299,L,108.52,c735eaf6de785f351429f0a70abc26d9
Normal-5734.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5734.png,Normal/images/Normal-5734.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5734.png,Normal/masks/Normal-5734.png,True,False,299,299,L,100.6,ad2324fedff1db36a723c49634d9f4c1
Normal-5735.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5735.png,Normal/images/Normal-5735.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5735.png,Normal/masks/Normal-5735.png,True,False,299,299,L,169.31,82fb3e92e4b30bad993d37aede5ace6a
Normal-5736.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5736.png,Normal/images/Normal-5736.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5736.png,Normal/masks/Normal-5736.png,True,False,299,299,L,150.9,2466cb413e0a180471c51302fdd1a32d
Normal-5737.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5737.png,Normal/images/Normal-5737.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5737.png,Normal/masks/Normal-5737.png,True,False,299,299,L,112.5,33f51ab5eaa58af1bc73e91d0113534e
Normal-5738.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5738.png,Normal/images/Normal-5738.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5738.png,Normal/masks/Normal-5738.png,True,False,299,299,L,123.73,6f8fd1e839801202add1f3fb47885a26
Normal-5739.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5739.png,Normal/images/Normal-5739.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5739.png,Normal/masks/Normal-5739.png,True,False,299,299,L,154.34,bb560e2a26ccadebf4411e6fbbb89d60
Normal-574.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-574.png,Normal/images/Normal-574.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-574.png,Normal/masks/Normal-574.png,True,False,299,299,L,116.48,33803a3c34e0765fa648d116b01fcf2f
Normal-5740.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5740.png,Normal/images/Normal-5740.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5740.png,Normal/masks/Normal-5740.png,True,False,299,299,L,182.38,73106efc765e670cc958cfc4360cae63
Normal-5741.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5741.png,Normal/images/Normal-5741.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5741.png,Normal/masks/Normal-5741.png,True,False,299,299,L,112.06,8301187d27ab3835c9eb064f6c0f4acb
Normal-5742.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5742.png,Normal/images/Normal-5742.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5742.png,Normal/masks/Normal-5742.png,True,False,299,299,L,164.44,3e2e373fccdcae8fdfb92b814dbd8c92
Normal-5743.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5743.png,Normal/images/Normal-5743.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5743.png,Normal/masks/Normal-5743.png,True,False,299,299,L,114.12,277464b327bbb8491caa4f69828885e2
Normal-5744.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5744.png,Normal/images/Normal-5744.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5744.png,Normal/masks/Normal-5744.png,True,False,299,299,L,176.22,503acf7ab5fcdd9cf5d3ecc4eac6d16f
Normal-5745.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5745.png,Normal/images/Normal-5745.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5745.png,Normal/masks/Normal-5745.png,True,False,299,299,L,146.42,7b807254cd50fb37521e28b49df5cadd
Normal-5746.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5746.png,Normal/images/Normal-5746.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5746.png,Normal/masks/Normal-5746.png,True,False,299,299,L,122.5,ca17c710aa0468390575ffef459d6797
Normal-5747.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5747.png,Normal/images/Normal-5747.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5747.png,Normal/masks/Normal-5747.png,True,False,299,299,L,154.49,fd5040377a9b40d10b09b39aa40a31b4
Normal-5748.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5748.png,Normal/images/Normal-5748.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5748.png,Normal/masks/Normal-5748.png,True,False,299,299,L,104.71,46bd804be9eb6183083f7b671ffd67b7
Normal-5749.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5749.png,Normal/images/Normal-5749.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5749.png,Normal/masks/Normal-5749.png,True,False,299,299,L,158.69,827c0079ce8e892a29786b5c13139296
Normal-575.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-575.png,Normal/images/Normal-575.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-575.png,Normal/masks/Normal-575.png,True,False,299,299,L,124.37,ce46eeb37086182e1f53426db9305acb
Normal-5750.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5750.png,Normal/images/Normal-5750.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5750.png,Normal/masks/Normal-5750.png,True,False,299,299,L,124.36,ed1a5c3448a772dac3824e85d1ed4bdf
Normal-5751.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5751.png,Normal/images/Normal-5751.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5751.png,Normal/masks/Normal-5751.png,True,False,299,299,L,141.89,f390e0a730ee1595a9b2c9f9d3ffe9cb
Normal-5752.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5752.png,Normal/images/Normal-5752.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5752.png,Normal/masks/Normal-5752.png,True,False,299,299,L,125.9,2af55016974000b8dcc28e584756652c
Normal-5753.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5753.png,Normal/images/Normal-5753.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5753.png,Normal/masks/Normal-5753.png,True,False,299,299,L,114.5,5b0d3a3555d58defddbb4dc22b5dafa0
Normal-5754.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5754.png,Normal/images/Normal-5754.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5754.png,Normal/masks/Normal-5754.png,True,False,299,299,L,151.27,e8a5d27147db0f481f5d70167fe2682b
Normal-5755.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5755.png,Normal/images/Normal-5755.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5755.png,Normal/masks/Normal-5755.png,True,False,299,299,L,123.95,261e2986c67c890fca1b5e3a093846cc
Normal-5756.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5756.png,Normal/images/Normal-5756.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5756.png,Normal/masks/Normal-5756.png,True,False,299,299,L,128.96,7835fbdaaf59315dd513b8d77971922d
Normal-5757.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5757.png,Normal/images/Normal-5757.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5757.png,Normal/masks/Normal-5757.png,True,False,299,299,L,132.81,d77b37ff57c71700014f226d2f932fa6
Normal-5758.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5758.png,Normal/images/Normal-5758.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5758.png,Normal/masks/Normal-5758.png,True,False,299,299,L,116.61,4fdf1369f7885191d126dd1701af747c
Normal-5759.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5759.png,Normal/images/Normal-5759.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5759.png,Normal/masks/Normal-5759.png,True,False,299,299,L,145.69,76bfa797f26b54894ee797a664215341
Normal-576.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-576.png,Normal/images/Normal-576.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-576.png,Normal/masks/Normal-576.png,True,False,299,299,L,118.15,3362982f4ea895f39c43b0f6d86ac423
Normal-5760.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5760.png,Normal/images/Normal-5760.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5760.png,Normal/masks/Normal-5760.png,True,False,299,299,L,141.27,35ecfa74eda2eb00722df7892de94cfc
Normal-5761.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5761.png,Normal/images/Normal-5761.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5761.png,Normal/masks/Normal-5761.png,True,False,299,299,L,115.9,f23b821ffaf9d0c47a7d1e623dd8e65f
Normal-5762.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5762.png,Normal/images/Normal-5762.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5762.png,Normal/masks/Normal-5762.png,True,False,299,299,L,118.57,ef139257f904781dd4e9d3e40c5b26d1
Normal-5763.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5763.png,Normal/images/Normal-5763.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5763.png,Normal/masks/Normal-5763.png,True,False,299,299,L,171.92,00513aec922fe37174deee244db9765c
Normal-5764.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5764.png,Normal/images/Normal-5764.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5764.png,Normal/masks/Normal-5764.png,True,False,299,299,L,52.77,097dec04d856e686d02e3c1a31468e91
Normal-5765.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5765.png,Normal/images/Normal-5765.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5765.png,Normal/masks/Normal-5765.png,True,False,299,299,L,180.12,edf7e615b5efd17a93becd4801af2e2c
Normal-5766.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5766.png,Normal/images/Normal-5766.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5766.png,Normal/masks/Normal-5766.png,True,False,299,299,L,164.63,19ffe0103e2cefe0110ea80ce9292ecc
Normal-5767.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5767.png,Normal/images/Normal-5767.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5767.png,Normal/masks/Normal-5767.png,True,False,299,299,L,137.33,24ddc8549fab24c15d54f7be0c4b800b
Normal-5768.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5768.png,Normal/images/Normal-5768.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5768.png,Normal/masks/Normal-5768.png,True,False,299,299,L,102.53,5ed38471300d2c33af8d4f3c4cade2f6
Normal-5769.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5769.png,Normal/images/Normal-5769.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5769.png,Normal/masks/Normal-5769.png,True,False,299,299,L,122.74,25306a32d0879ee1006b3ae28a7b9c88
Normal-577.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-577.png,Normal/images/Normal-577.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-577.png,Normal/masks/Normal-577.png,True,False,299,299,L,124.62,5652368e08ec55364823e9616e801bd5
Normal-5770.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5770.png,Normal/images/Normal-5770.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5770.png,Normal/masks/Normal-5770.png,True,False,299,299,L,109.13,58b010c1d81cea399559a600f7ce021c
Normal-5771.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5771.png,Normal/images/Normal-5771.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5771.png,Normal/masks/Normal-5771.png,True,False,299,299,L,114.47,ef120917f97f0525a53667830bcb4500
Normal-5772.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5772.png,Normal/images/Normal-5772.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5772.png,Normal/masks/Normal-5772.png,True,False,299,299,L,129.19,2cc290512ef47c17c274f688ebd08cf8
Normal-5773.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5773.png,Normal/images/Normal-5773.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5773.png,Normal/masks/Normal-5773.png,True,False,299,299,L,113.07,dbfb2e7ae95dc2fe41d691f6cdc807f9
Normal-5774.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5774.png,Normal/images/Normal-5774.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5774.png,Normal/masks/Normal-5774.png,True,False,299,299,L,148.77,541b17299a8f3385dccff8077792e2e3
Normal-5775.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5775.png,Normal/images/Normal-5775.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5775.png,Normal/masks/Normal-5775.png,True,False,299,299,L,108.81,9a411a43b3d06cd337d621b00ca23055
Normal-5776.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5776.png,Normal/images/Normal-5776.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5776.png,Normal/masks/Normal-5776.png,True,False,299,299,L,116.45,237fc1d705e71e5373a2b9b4b2e6d851
Normal-5777.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5777.png,Normal/images/Normal-5777.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5777.png,Normal/masks/Normal-5777.png,True,False,299,299,L,138.02,b8ae228b013e4d5e0192861c410712cb
Normal-5778.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5778.png,Normal/images/Normal-5778.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5778.png,Normal/masks/Normal-5778.png,True,False,299,299,L,123.22,ccc4f5c0d456ea4321d12515cbd1f4dc
Normal-5779.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5779.png,Normal/images/Normal-5779.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5779.png,Normal/masks/Normal-5779.png,True,False,299,299,L,102.91,3552ae72bd08c6f925990a9663c4aab2
Normal-578.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-578.png,Normal/images/Normal-578.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-578.png,Normal/masks/Normal-578.png,True,False,299,299,L,133.54,cf1af6fabb955cf1a7c360a43e83cc3f
Normal-5780.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5780.png,Normal/images/Normal-5780.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5780.png,Normal/masks/Normal-5780.png,True,False,299,299,L,172.99,70ec1456bd8f799bc0fa30008c555216
Normal-5781.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5781.png,Normal/images/Normal-5781.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5781.png,Normal/masks/Normal-5781.png,True,False,299,299,L,114.9,5549d3a1bad31454598c1c0f76ea89f4
Normal-5782.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5782.png,Normal/images/Normal-5782.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5782.png,Normal/masks/Normal-5782.png,True,False,299,299,L,124.15,2d6e7f26e4ee5d99ff1788c45d944cf3
Normal-5783.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5783.png,Normal/images/Normal-5783.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5783.png,Normal/masks/Normal-5783.png,True,False,299,299,L,122.2,737891a6a2b8bca0bc392d6133dde0eb
Normal-5784.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5784.png,Normal/images/Normal-5784.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5784.png,Normal/masks/Normal-5784.png,True,False,299,299,L,122.48,b334046a04b8a975c84ae1a4ed08b5a4
Normal-5785.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5785.png,Normal/images/Normal-5785.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5785.png,Normal/masks/Normal-5785.png,True,False,299,299,L,112.9,709fc6f3d26bf69e798b8dd9e513b1b8
Normal-5786.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5786.png,Normal/images/Normal-5786.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5786.png,Normal/masks/Normal-5786.png,True,False,299,299,L,105.96,0e4252298ec06f161aa0fd20cf5fd744
Normal-5787.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5787.png,Normal/images/Normal-5787.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5787.png,Normal/masks/Normal-5787.png,True,False,299,299,L,120.34,2d66bfa66da8920d33dfceb7359e3119
Normal-5788.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5788.png,Normal/images/Normal-5788.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5788.png,Normal/masks/Normal-5788.png,True,False,299,299,L,165.64,b5dfd98623e2ceacf108e620a8ce6dc0
Normal-5789.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5789.png,Normal/images/Normal-5789.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5789.png,Normal/masks/Normal-5789.png,True,False,299,299,L,123.28,6b70505ff38df5751c13160a348d161a
Normal-579.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-579.png,Normal/images/Normal-579.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-579.png,Normal/masks/Normal-579.png,True,False,299,299,L,119.77,19ea03a145439780461323a313b3df26
Normal-5790.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5790.png,Normal/images/Normal-5790.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5790.png,Normal/masks/Normal-5790.png,True,False,299,299,L,129.15,217f579bf34d49207db2f595934c82d0
Normal-5791.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5791.png,Normal/images/Normal-5791.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5791.png,Normal/masks/Normal-5791.png,True,False,299,299,L,156.28,c872eeca2898420292c8cb5a09ee831c
Normal-5792.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5792.png,Normal/images/Normal-5792.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5792.png,Normal/masks/Normal-5792.png,True,False,299,299,L,103.87,34e05397996bc7fc92deb6e8b05f47f7
Normal-5793.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5793.png,Normal/images/Normal-5793.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5793.png,Normal/masks/Normal-5793.png,True,False,299,299,L,112.1,7ac3c1cd36c7808955923b57da2d4733
Normal-5794.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5794.png,Normal/images/Normal-5794.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5794.png,Normal/masks/Normal-5794.png,True,False,299,299,L,115.44,6b541f8e16c71ce8b5e4c765154ad2c1
Normal-5795.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5795.png,Normal/images/Normal-5795.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5795.png,Normal/masks/Normal-5795.png,True,False,299,299,L,128.65,f0a507137f9c884bc5d7725fe418cfad
Normal-5796.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5796.png,Normal/images/Normal-5796.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5796.png,Normal/masks/Normal-5796.png,True,False,299,299,L,130.74,b23f76aa5ff7b9eb5ac9457b2e2f5873
Normal-5797.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5797.png,Normal/images/Normal-5797.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5797.png,Normal/masks/Normal-5797.png,True,False,299,299,L,106.05,5faa563defea490e5130cb5d0ef5a880
Normal-5798.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5798.png,Normal/images/Normal-5798.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5798.png,Normal/masks/Normal-5798.png,True,False,299,299,L,114.05,3f8409955e713a9c3a38f6c297941324
Normal-5799.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5799.png,Normal/images/Normal-5799.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5799.png,Normal/masks/Normal-5799.png,True,False,299,299,L,123.74,0a51cc217f7832639fba989c899fc0e7
Normal-58.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-58.png,Normal/images/Normal-58.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-58.png,Normal/masks/Normal-58.png,True,False,299,299,L,126.57,709aa725da31b5e94d4d6290e3b0af20
Normal-580.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-580.png,Normal/images/Normal-580.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-580.png,Normal/masks/Normal-580.png,True,False,299,299,L,127.33,8114b118a5165d3e0386ff875ceb6eb5
Normal-5800.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5800.png,Normal/images/Normal-5800.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5800.png,Normal/masks/Normal-5800.png,True,False,299,299,L,119.06,f451d57fa176114cd54b20fb9783d183
Normal-5801.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5801.png,Normal/images/Normal-5801.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5801.png,Normal/masks/Normal-5801.png,True,False,299,299,L,102.11,f1dc1805acc680def63c93ce0c35f032
Normal-5802.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5802.png,Normal/images/Normal-5802.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5802.png,Normal/masks/Normal-5802.png,True,False,299,299,L,121.49,a9ec1a069e72924aa2dc569d0bf36b45
Normal-5803.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5803.png,Normal/images/Normal-5803.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5803.png,Normal/masks/Normal-5803.png,True,False,299,299,L,169.97,ca275459e0af716dd31bd19b7950d528
Normal-5804.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5804.png,Normal/images/Normal-5804.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5804.png,Normal/masks/Normal-5804.png,True,False,299,299,L,149.48,a45104c9656671b9446adbbbb39867d6
Normal-5805.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5805.png,Normal/images/Normal-5805.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5805.png,Normal/masks/Normal-5805.png,True,False,299,299,L,114.12,d36f1b29cfd8572ae62fa6c9fee9c47a
Normal-5806.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5806.png,Normal/images/Normal-5806.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5806.png,Normal/masks/Normal-5806.png,True,False,299,299,L,102.44,c58c12c1524272d1ed29e265ac7ef0e2
Normal-5807.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5807.png,Normal/images/Normal-5807.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5807.png,Normal/masks/Normal-5807.png,True,False,299,299,L,131.38,00dcbaa11cc29242758d0c1c85a5937c
Normal-5808.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5808.png,Normal/images/Normal-5808.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5808.png,Normal/masks/Normal-5808.png,True,False,299,299,L,100.83,82b5ea9df60d351a0f13abff0191482b
Normal-5809.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5809.png,Normal/images/Normal-5809.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5809.png,Normal/masks/Normal-5809.png,True,False,299,299,L,107.55,759b32c3a1899bbc96e965538b795191
Normal-581.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-581.png,Normal/images/Normal-581.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-581.png,Normal/masks/Normal-581.png,True,False,299,299,L,112.74,559eded84de61825eeb165c9c0ef673a
Normal-5810.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5810.png,Normal/images/Normal-5810.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5810.png,Normal/masks/Normal-5810.png,True,False,299,299,L,98.69,552523539d557747cbcaf9b21addf4de
Normal-5811.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5811.png,Normal/images/Normal-5811.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5811.png,Normal/masks/Normal-5811.png,True,False,299,299,L,121.38,32411858938bd329879f00961ffb67ea
Normal-5812.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5812.png,Normal/images/Normal-5812.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5812.png,Normal/masks/Normal-5812.png,True,False,299,299,L,127.66,17a9d9896b6bbb223d3736151cd1ea96
Normal-5813.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5813.png,Normal/images/Normal-5813.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5813.png,Normal/masks/Normal-5813.png,True,False,299,299,L,90.81,5c07947fed5aab0e61a95c3d526322eb
Normal-5814.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5814.png,Normal/images/Normal-5814.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5814.png,Normal/masks/Normal-5814.png,True,False,299,299,L,119.96,2c0f8366dd8e8b0870d0ef3db3b67ffd
Normal-5815.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5815.png,Normal/images/Normal-5815.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5815.png,Normal/masks/Normal-5815.png,True,False,299,299,L,127.73,60a9857d617d609beb0a485b7f0d2b46
Normal-5816.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5816.png,Normal/images/Normal-5816.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5816.png,Normal/masks/Normal-5816.png,True,False,299,299,L,146.23,ca5ee79c005f0f58a87ef987b9e183de
Normal-5817.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5817.png,Normal/images/Normal-5817.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5817.png,Normal/masks/Normal-5817.png,True,False,299,299,L,112.7,780ca734eeed9c213cbe7be28a50246b
Normal-5818.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5818.png,Normal/images/Normal-5818.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5818.png,Normal/masks/Normal-5818.png,True,False,299,299,L,97.55,5c38dd973427e2147fc479caa1ed6513
Normal-5819.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5819.png,Normal/images/Normal-5819.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5819.png,Normal/masks/Normal-5819.png,True,False,299,299,L,121.68,2ef16c559204ae5ffb70674b4eb25617
Normal-582.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-582.png,Normal/images/Normal-582.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-582.png,Normal/masks/Normal-582.png,True,False,299,299,L,120.53,f9a312e5a1ede56a3d4b95992d64d157
Normal-5820.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5820.png,Normal/images/Normal-5820.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5820.png,Normal/masks/Normal-5820.png,True,False,299,299,L,147.05,91bede3a651fdbb0fba523b310929a7a
Normal-5821.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5821.png,Normal/images/Normal-5821.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5821.png,Normal/masks/Normal-5821.png,True,False,299,299,L,133.53,6a17949bb11a5e5f4f8a563820991b76
Normal-5822.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5822.png,Normal/images/Normal-5822.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5822.png,Normal/masks/Normal-5822.png,True,False,299,299,L,158.24,e23fe6de0728238e1869fbd00412155b
Normal-5823.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5823.png,Normal/images/Normal-5823.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5823.png,Normal/masks/Normal-5823.png,True,False,299,299,L,149.35,208ac1ad50f2d12fbc12b9120c4a783b
Normal-5824.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5824.png,Normal/images/Normal-5824.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5824.png,Normal/masks/Normal-5824.png,True,False,299,299,L,159.91,b8cbc13a968388855d4191908f9ced40
Normal-5825.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5825.png,Normal/images/Normal-5825.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5825.png,Normal/masks/Normal-5825.png,True,False,299,299,L,159.01,f1d07612d6f4d3cb8f94c321c26b2611
Normal-5826.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5826.png,Normal/images/Normal-5826.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5826.png,Normal/masks/Normal-5826.png,True,False,299,299,L,129.34,9a9a469d796a3b4e01e0b049ca25754b
Normal-5827.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5827.png,Normal/images/Normal-5827.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5827.png,Normal/masks/Normal-5827.png,True,False,299,299,L,111.81,d238620b3b18ab5eb830e39a321d68b1
Normal-5828.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5828.png,Normal/images/Normal-5828.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5828.png,Normal/masks/Normal-5828.png,True,False,299,299,L,170.07,b2a943cf76ee8f16ec893bbd68ba9018
Normal-5829.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5829.png,Normal/images/Normal-5829.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5829.png,Normal/masks/Normal-5829.png,True,False,299,299,L,112.17,c9fb8e581198842633526a7af717e92c
Normal-583.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-583.png,Normal/images/Normal-583.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-583.png,Normal/masks/Normal-583.png,True,False,299,299,L,116.96,32c57bbd4d007a528f226f23e9351a36
Normal-5830.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5830.png,Normal/images/Normal-5830.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5830.png,Normal/masks/Normal-5830.png,True,False,299,299,L,135.39,e4f7681cd840b37236866bac9b62760c
Normal-5831.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5831.png,Normal/images/Normal-5831.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5831.png,Normal/masks/Normal-5831.png,True,False,299,299,L,166.18,885336d5189eaf98c0c24e14977399e6
Normal-5832.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5832.png,Normal/images/Normal-5832.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5832.png,Normal/masks/Normal-5832.png,True,False,299,299,L,119.57,349e3f23a46102c9d87d682487dee0b4
Normal-5833.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5833.png,Normal/images/Normal-5833.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5833.png,Normal/masks/Normal-5833.png,True,False,299,299,L,106.44,8f23e5b93318f255e47afce301443b77
Normal-5834.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5834.png,Normal/images/Normal-5834.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5834.png,Normal/masks/Normal-5834.png,True,False,299,299,L,116.8,f782e9d411bbacac34abec1cbd543f90
Normal-5835.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5835.png,Normal/images/Normal-5835.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5835.png,Normal/masks/Normal-5835.png,True,False,299,299,L,163.98,baae184f178bc53f3d1674f42e058fa7
Normal-5836.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5836.png,Normal/images/Normal-5836.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5836.png,Normal/masks/Normal-5836.png,True,False,299,299,L,125.08,e4cea584367b4dc3dff1963b2cc0164f
Normal-5837.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5837.png,Normal/images/Normal-5837.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5837.png,Normal/masks/Normal-5837.png,True,False,299,299,L,105.26,d7c46a17c5ae53144a8c4daf232e9c70
Normal-5838.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5838.png,Normal/images/Normal-5838.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5838.png,Normal/masks/Normal-5838.png,True,False,299,299,L,114.47,479984c147552e30668182e6dda141cb
Normal-5839.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5839.png,Normal/images/Normal-5839.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5839.png,Normal/masks/Normal-5839.png,True,False,299,299,L,151.23,4d843e77bf870d01b533af733526894d
Normal-584.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-584.png,Normal/images/Normal-584.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-584.png,Normal/masks/Normal-584.png,True,False,299,299,L,116.57,2d3d807330a8add2f023f8e1374e9ae7
Normal-5840.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5840.png,Normal/images/Normal-5840.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5840.png,Normal/masks/Normal-5840.png,True,False,299,299,L,170.56,e28612352cb45b2a4617360d05448245
Normal-5841.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5841.png,Normal/images/Normal-5841.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5841.png,Normal/masks/Normal-5841.png,True,False,299,299,L,172.85,d6b521099d864905bb6e5f1b8947dba4
Normal-5842.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5842.png,Normal/images/Normal-5842.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5842.png,Normal/masks/Normal-5842.png,True,False,299,299,L,128.05,451872200268fc4847bb4ee7267d6aa9
Normal-5843.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5843.png,Normal/images/Normal-5843.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5843.png,Normal/masks/Normal-5843.png,True,False,299,299,L,130.85,b23fec4765a6805905f5a69855ad1a32
Normal-5844.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5844.png,Normal/images/Normal-5844.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5844.png,Normal/masks/Normal-5844.png,True,False,299,299,L,119.33,9c631669c627f6d37b3747d434fa6ec6
Normal-5845.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5845.png,Normal/images/Normal-5845.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5845.png,Normal/masks/Normal-5845.png,True,False,299,299,L,102.81,4cc1017d6637d92ecb57a041980407d3
Normal-5846.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5846.png,Normal/images/Normal-5846.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5846.png,Normal/masks/Normal-5846.png,True,False,299,299,L,67.43,ffd8a9d6d7f2ba5d545a2d7ee89614c4
Normal-5847.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5847.png,Normal/images/Normal-5847.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5847.png,Normal/masks/Normal-5847.png,True,False,299,299,L,117.04,ea6478f6d4ca717030c1fa39a09be753
Normal-5848.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5848.png,Normal/images/Normal-5848.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5848.png,Normal/masks/Normal-5848.png,True,False,299,299,L,121.11,484db14b615b33b5020a4f85dbf4e57f
Normal-5849.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5849.png,Normal/images/Normal-5849.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5849.png,Normal/masks/Normal-5849.png,True,False,299,299,L,146.55,c87797dcfd365e4d9aa7e3c707f42217
Normal-585.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-585.png,Normal/images/Normal-585.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-585.png,Normal/masks/Normal-585.png,True,False,299,299,L,134.86,27fdf0b63586b43a1daa80699bc13109
Normal-5850.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5850.png,Normal/images/Normal-5850.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5850.png,Normal/masks/Normal-5850.png,True,False,299,299,L,130.3,b7191803b76bc92820f28f4ed54ceeee
Normal-5851.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5851.png,Normal/images/Normal-5851.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5851.png,Normal/masks/Normal-5851.png,True,False,299,299,L,158.17,4a442148768262a59be56c0b924db900
Normal-5852.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5852.png,Normal/images/Normal-5852.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5852.png,Normal/masks/Normal-5852.png,True,False,299,299,L,178.5,cdbe19e37ee6c641018cff368043c17b
Normal-5853.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5853.png,Normal/images/Normal-5853.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5853.png,Normal/masks/Normal-5853.png,True,False,299,299,L,121.84,c62d3a2c8948ed7d040737e2c57bca07
Normal-5854.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5854.png,Normal/images/Normal-5854.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5854.png,Normal/masks/Normal-5854.png,True,False,299,299,L,120.52,05842f64625a13f27261c74aaf41462b
Normal-5855.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5855.png,Normal/images/Normal-5855.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5855.png,Normal/masks/Normal-5855.png,True,False,299,299,L,159.27,4c36ea3902a1d351ecaa875da95583b8
Normal-5856.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5856.png,Normal/images/Normal-5856.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5856.png,Normal/masks/Normal-5856.png,True,False,299,299,L,147.96,9d6360b89fad22538d803a56bd856900
Normal-5857.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5857.png,Normal/images/Normal-5857.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5857.png,Normal/masks/Normal-5857.png,True,False,299,299,L,140.49,bdc99cba0a4af5d6eebc9f8e409c3dc9
Normal-5858.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5858.png,Normal/images/Normal-5858.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5858.png,Normal/masks/Normal-5858.png,True,False,299,299,L,115.41,91e3a3568bd15f4b3be8b48985e3e839
Normal-5859.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5859.png,Normal/images/Normal-5859.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5859.png,Normal/masks/Normal-5859.png,True,False,299,299,L,128.77,d5a9ee78c5e531f140e5a27649c4ed1c
Normal-586.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-586.png,Normal/images/Normal-586.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-586.png,Normal/masks/Normal-586.png,True,False,299,299,L,119.3,9d7ef92110b4f0bc5d436acd5054ca4d
Normal-5860.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5860.png,Normal/images/Normal-5860.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5860.png,Normal/masks/Normal-5860.png,True,False,299,299,L,144.1,ba5ce0629f7155a274a0253e9aea63b2
Normal-5861.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5861.png,Normal/images/Normal-5861.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5861.png,Normal/masks/Normal-5861.png,True,False,299,299,L,115.4,331d04c807dca544abb2b959743a6667
Normal-5862.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5862.png,Normal/images/Normal-5862.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5862.png,Normal/masks/Normal-5862.png,True,False,299,299,L,150.06,e83c4b7125e738738fb68828c9b250cb
Normal-5863.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5863.png,Normal/images/Normal-5863.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5863.png,Normal/masks/Normal-5863.png,True,False,299,299,L,135.29,3af494a24909bde530ed7be4d547f8f9
Normal-5864.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5864.png,Normal/images/Normal-5864.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5864.png,Normal/masks/Normal-5864.png,True,False,299,299,L,119.43,af0742c91a79e50160e61fd191f7e6cd
Normal-5865.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5865.png,Normal/images/Normal-5865.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5865.png,Normal/masks/Normal-5865.png,True,False,299,299,L,111.95,db01e2d2049080bfbb01f34d27c61a3e
Normal-5866.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5866.png,Normal/images/Normal-5866.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5866.png,Normal/masks/Normal-5866.png,True,False,299,299,L,141.81,164d341942d7c6683ec8b29d7ba15779
Normal-5867.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5867.png,Normal/images/Normal-5867.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5867.png,Normal/masks/Normal-5867.png,True,False,299,299,L,124.28,79c3c9c69afbbc0cbd22a5db84447b5b
Normal-5868.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5868.png,Normal/images/Normal-5868.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5868.png,Normal/masks/Normal-5868.png,True,False,299,299,L,149.66,2974f6241c806ff30e4091d9529b3dc2
Normal-5869.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5869.png,Normal/images/Normal-5869.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5869.png,Normal/masks/Normal-5869.png,True,False,299,299,L,170.18,a143655912e500c82ff2a3bd3c2f0cc9
Normal-587.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-587.png,Normal/images/Normal-587.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-587.png,Normal/masks/Normal-587.png,True,False,299,299,L,122.21,a526f5d55094a5cbc4b9aab3e2fa8bb3
Normal-5870.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5870.png,Normal/images/Normal-5870.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5870.png,Normal/masks/Normal-5870.png,True,False,299,299,L,130.81,b8a0daf1b579b6aaa693fcfa5e1f0d13
Normal-5871.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5871.png,Normal/images/Normal-5871.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5871.png,Normal/masks/Normal-5871.png,True,False,299,299,L,107.67,4a6ba0258d33d3866b011b088661c5d9
Normal-5872.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5872.png,Normal/images/Normal-5872.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5872.png,Normal/masks/Normal-5872.png,True,False,299,299,L,123.75,76d94c3b6f933cfd3898d91232c7854f
Normal-5873.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5873.png,Normal/images/Normal-5873.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5873.png,Normal/masks/Normal-5873.png,True,False,299,299,L,141.81,914490d720bc1e8189fb9cef9177c6ef
Normal-5874.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5874.png,Normal/images/Normal-5874.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5874.png,Normal/masks/Normal-5874.png,True,False,299,299,L,117.96,4c074b90c11f5976220d6917825af63b
Normal-5875.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5875.png,Normal/images/Normal-5875.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5875.png,Normal/masks/Normal-5875.png,True,False,299,299,L,154.14,6a87fb564dae5e7f1b932cfeac220666
Normal-5876.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5876.png,Normal/images/Normal-5876.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5876.png,Normal/masks/Normal-5876.png,True,False,299,299,L,172.6,18f4b80fbb9381feb0f74224e4fd34e8
Normal-5877.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5877.png,Normal/images/Normal-5877.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5877.png,Normal/masks/Normal-5877.png,True,False,299,299,L,160.58,cb422b355d280abd86ade0691320dd73
Normal-5878.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5878.png,Normal/images/Normal-5878.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5878.png,Normal/masks/Normal-5878.png,True,False,299,299,L,138.72,a235d918d1ba4df3b65d38af9d94e21e
Normal-5879.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5879.png,Normal/images/Normal-5879.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5879.png,Normal/masks/Normal-5879.png,True,False,299,299,L,120.57,5aaaddee1cbbc0bb1f8cc807a9df0503
Normal-588.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-588.png,Normal/images/Normal-588.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-588.png,Normal/masks/Normal-588.png,True,False,299,299,L,119.46,3a62a0bf50ed0fab1d99c3c1829b81fe
Normal-5880.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5880.png,Normal/images/Normal-5880.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5880.png,Normal/masks/Normal-5880.png,True,False,299,299,L,129.29,8e71a21db452a6a02d475b7fd3db6867
Normal-5881.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5881.png,Normal/images/Normal-5881.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5881.png,Normal/masks/Normal-5881.png,True,False,299,299,L,127.02,7dcf1fb8895f4ee61803d56f0ddff732
Normal-5882.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5882.png,Normal/images/Normal-5882.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5882.png,Normal/masks/Normal-5882.png,True,False,299,299,L,91.51,d103d4950ac4e06392825d927107f61a
Normal-5883.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5883.png,Normal/images/Normal-5883.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5883.png,Normal/masks/Normal-5883.png,True,False,299,299,L,106.33,16d1e5c8bc47dd97b2d8b264d69a9ba3
Normal-5884.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5884.png,Normal/images/Normal-5884.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5884.png,Normal/masks/Normal-5884.png,True,False,299,299,L,125.12,282b0a59e069a9030763d0d34ae60293
Normal-5885.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5885.png,Normal/images/Normal-5885.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5885.png,Normal/masks/Normal-5885.png,True,False,299,299,L,157.84,e76e2d27d2deb85c9cad651fb9801289
Normal-5886.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5886.png,Normal/images/Normal-5886.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5886.png,Normal/masks/Normal-5886.png,True,False,299,299,L,96.28,e228ca132c864ed2fe2603061b7a8f19
Normal-5887.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5887.png,Normal/images/Normal-5887.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5887.png,Normal/masks/Normal-5887.png,True,False,299,299,L,84.63,45f20d57eea22a93e3634a586c1152c0
Normal-5888.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5888.png,Normal/images/Normal-5888.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5888.png,Normal/masks/Normal-5888.png,True,False,299,299,L,117.31,018333ed08d63811cd512d535f1b8934
Normal-5889.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5889.png,Normal/images/Normal-5889.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5889.png,Normal/masks/Normal-5889.png,True,False,299,299,L,144.21,c498f9560356e96ae490c87079d0856d
Normal-589.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-589.png,Normal/images/Normal-589.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-589.png,Normal/masks/Normal-589.png,True,False,299,299,L,135.95,c326c2698752071285fc7834c68df066
Normal-5890.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5890.png,Normal/images/Normal-5890.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5890.png,Normal/masks/Normal-5890.png,True,False,299,299,L,130.44,ef6a27a572b5a75457ececab80046422
Normal-5891.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5891.png,Normal/images/Normal-5891.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5891.png,Normal/masks/Normal-5891.png,True,False,299,299,L,141.6,2be1fa85044ec7f448037eb6e6c245af
Normal-5892.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5892.png,Normal/images/Normal-5892.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5892.png,Normal/masks/Normal-5892.png,True,False,299,299,L,173.72,8b7e6361787d1967ddb81de9badf291c
Normal-5893.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5893.png,Normal/images/Normal-5893.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5893.png,Normal/masks/Normal-5893.png,True,False,299,299,L,123.18,269eee3d3b0e5f8ce2a6948e561dd9ca
Normal-5894.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5894.png,Normal/images/Normal-5894.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5894.png,Normal/masks/Normal-5894.png,True,False,299,299,L,123.26,345d547d2fbe031a0ed26a22dded2593
Normal-5895.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5895.png,Normal/images/Normal-5895.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5895.png,Normal/masks/Normal-5895.png,True,False,299,299,L,178.63,a60d6c22c81e7a2cf3aae95211ba012e
Normal-5896.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5896.png,Normal/images/Normal-5896.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5896.png,Normal/masks/Normal-5896.png,True,False,299,299,L,116.48,171f1384c3643383580e192b2987fe76
Normal-5897.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5897.png,Normal/images/Normal-5897.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5897.png,Normal/masks/Normal-5897.png,True,False,299,299,L,112.31,fca8682cc677c49be3999ed0d89c2a44
Normal-5898.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5898.png,Normal/images/Normal-5898.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5898.png,Normal/masks/Normal-5898.png,True,False,299,299,L,116.21,9b4200eacbd2f180de4d4d5ab4dc5933
Normal-5899.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5899.png,Normal/images/Normal-5899.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5899.png,Normal/masks/Normal-5899.png,True,False,299,299,L,101.2,e084b78f5acf0f5d636755aac3c12c23
Normal-59.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-59.png,Normal/images/Normal-59.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-59.png,Normal/masks/Normal-59.png,True,False,299,299,L,91.23,9ac8fc82a478c1cd2e7a2a030c238fe7
Normal-590.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-590.png,Normal/images/Normal-590.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-590.png,Normal/masks/Normal-590.png,True,False,299,299,L,137.08,f1716b875df6e239cbf1ad8c36206795
Normal-5900.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5900.png,Normal/images/Normal-5900.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5900.png,Normal/masks/Normal-5900.png,True,False,299,299,L,132.09,313e71b6f947ebc7ddd54aa628b49060
Normal-5901.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5901.png,Normal/images/Normal-5901.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5901.png,Normal/masks/Normal-5901.png,True,False,299,299,L,130.95,02b158af35af5e212c5a75a9e1a450ae
Normal-5902.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5902.png,Normal/images/Normal-5902.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5902.png,Normal/masks/Normal-5902.png,True,False,299,299,L,103.6,c51e4a4a288ae5a9f6a8a0355d5fb6b7
Normal-5903.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5903.png,Normal/images/Normal-5903.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5903.png,Normal/masks/Normal-5903.png,True,False,299,299,L,138.03,2c674d3186934dce017778de41654b11
Normal-5904.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5904.png,Normal/images/Normal-5904.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5904.png,Normal/masks/Normal-5904.png,True,False,299,299,L,102.96,511a515dfdea855355fd3664cd77202e
Normal-5905.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5905.png,Normal/images/Normal-5905.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5905.png,Normal/masks/Normal-5905.png,True,False,299,299,L,108.17,555d6171dba28b750bf1f4c81abe33ce
Normal-5906.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5906.png,Normal/images/Normal-5906.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5906.png,Normal/masks/Normal-5906.png,True,False,299,299,L,73.56,63830447c915a665fc70ae5bd131f714
Normal-5907.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5907.png,Normal/images/Normal-5907.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5907.png,Normal/masks/Normal-5907.png,True,False,299,299,L,137.57,d1a1e655767a90bd16b616c1734bf05a
Normal-5908.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5908.png,Normal/images/Normal-5908.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5908.png,Normal/masks/Normal-5908.png,True,False,299,299,L,125.11,fc8e86826f7b6494c113101d48446cdb
Normal-5909.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5909.png,Normal/images/Normal-5909.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5909.png,Normal/masks/Normal-5909.png,True,False,299,299,L,113.2,d8e0e052d610bbb29b1950f99e34a427
Normal-591.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-591.png,Normal/images/Normal-591.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-591.png,Normal/masks/Normal-591.png,True,False,299,299,L,112.73,186348ac36640dcf400802dabbe5d176
Normal-5910.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5910.png,Normal/images/Normal-5910.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5910.png,Normal/masks/Normal-5910.png,True,False,299,299,L,80.75,ad6439a3ff6f26be454cd03281059cc7
Normal-5911.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5911.png,Normal/images/Normal-5911.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5911.png,Normal/masks/Normal-5911.png,True,False,299,299,L,149.84,b531a2d158005b7a6ff620d6dd391d21
Normal-5912.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5912.png,Normal/images/Normal-5912.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5912.png,Normal/masks/Normal-5912.png,True,False,299,299,L,147.66,904a822266d854a526799ec48bf6a433
Normal-5913.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5913.png,Normal/images/Normal-5913.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5913.png,Normal/masks/Normal-5913.png,True,False,299,299,L,113.69,f2a173c78b7162a9de1dd65534b60059
Normal-5914.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5914.png,Normal/images/Normal-5914.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5914.png,Normal/masks/Normal-5914.png,True,False,299,299,L,120.09,8cb55cc9d082c8cfe7673eff16f0083b
Normal-5915.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5915.png,Normal/images/Normal-5915.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5915.png,Normal/masks/Normal-5915.png,True,False,299,299,L,150.89,e4a58d9cfd2881f04b930894cbb7cf46
Normal-5916.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5916.png,Normal/images/Normal-5916.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5916.png,Normal/masks/Normal-5916.png,True,False,299,299,L,140.17,3867737ef5e9c3215936292ed6406094
Normal-5917.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5917.png,Normal/images/Normal-5917.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5917.png,Normal/masks/Normal-5917.png,True,False,299,299,L,118.64,29e5ec99a29da12158f97846d2aa89f1
Normal-5918.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5918.png,Normal/images/Normal-5918.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5918.png,Normal/masks/Normal-5918.png,True,False,299,299,L,150.92,671ec275493d6e8ed9999fb529db0b05
Normal-5919.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5919.png,Normal/images/Normal-5919.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5919.png,Normal/masks/Normal-5919.png,True,False,299,299,L,120.76,26d12343ab9b2ef91bdbf970ac146b83
Normal-592.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-592.png,Normal/images/Normal-592.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-592.png,Normal/masks/Normal-592.png,True,False,299,299,L,101.01,4448112f368db3a8ab6b1ee6f1568d11
Normal-5920.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5920.png,Normal/images/Normal-5920.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5920.png,Normal/masks/Normal-5920.png,True,False,299,299,L,127.02,f4fe291ab5108560416661af4aeecc83
Normal-5921.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5921.png,Normal/images/Normal-5921.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5921.png,Normal/masks/Normal-5921.png,True,False,299,299,L,120.27,d8fa27c30d1c8536752c22f987f54436
Normal-5922.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5922.png,Normal/images/Normal-5922.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5922.png,Normal/masks/Normal-5922.png,True,False,299,299,L,150.47,817f40d23ae21506917a62831290e6a0
Normal-5923.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5923.png,Normal/images/Normal-5923.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5923.png,Normal/masks/Normal-5923.png,True,False,299,299,L,129.08,f0f854a7a42bdf4278972a8c039813fc
Normal-5924.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5924.png,Normal/images/Normal-5924.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5924.png,Normal/masks/Normal-5924.png,True,False,299,299,L,107.64,d7da4e8b8d4e49e3347bdf2259715139
Normal-5925.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5925.png,Normal/images/Normal-5925.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5925.png,Normal/masks/Normal-5925.png,True,False,299,299,L,120.75,d850a90576d0632c7e74814dda961c0b
Normal-5926.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5926.png,Normal/images/Normal-5926.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5926.png,Normal/masks/Normal-5926.png,True,False,299,299,L,162.34,8196d68854905bc0797432235bc22c1f
Normal-5927.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5927.png,Normal/images/Normal-5927.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5927.png,Normal/masks/Normal-5927.png,True,False,299,299,L,100.59,895cb3a21d8237d77da67e3190f01759
Normal-5928.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5928.png,Normal/images/Normal-5928.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5928.png,Normal/masks/Normal-5928.png,True,False,299,299,L,106.96,688c5b42bdd22b20bd951549546a836c
Normal-5929.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5929.png,Normal/images/Normal-5929.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5929.png,Normal/masks/Normal-5929.png,True,False,299,299,L,159.66,3268b8771159eb6b36515859f3156909
Normal-593.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-593.png,Normal/images/Normal-593.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-593.png,Normal/masks/Normal-593.png,True,False,299,299,L,119.28,8851a619b05c223906e16d459a2d23bd
Normal-5930.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5930.png,Normal/images/Normal-5930.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5930.png,Normal/masks/Normal-5930.png,True,False,299,299,L,97.43,702349c30529eea37b97f9bb5a672a38
Normal-5931.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5931.png,Normal/images/Normal-5931.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5931.png,Normal/masks/Normal-5931.png,True,False,299,299,L,140.02,45b6c4db66b02a0739e28c784141baed
Normal-5932.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5932.png,Normal/images/Normal-5932.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5932.png,Normal/masks/Normal-5932.png,True,False,299,299,L,101.43,42e784fb33f001709ea0c1352efdbcbe
Normal-5933.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5933.png,Normal/images/Normal-5933.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5933.png,Normal/masks/Normal-5933.png,True,False,299,299,L,103.68,085b0e9191feea95948ea27f914207e9
Normal-5934.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5934.png,Normal/images/Normal-5934.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5934.png,Normal/masks/Normal-5934.png,True,False,299,299,L,118.29,0adb1d502f1746efec22a62af0eac89c
Normal-5935.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5935.png,Normal/images/Normal-5935.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5935.png,Normal/masks/Normal-5935.png,True,False,299,299,L,159.29,522958c5aa8d7725eb66ba3148db047c
Normal-5936.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5936.png,Normal/images/Normal-5936.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5936.png,Normal/masks/Normal-5936.png,True,False,299,299,L,121.81,ded7fd7bfdc81035b17393832a0cfd47
Normal-5937.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5937.png,Normal/images/Normal-5937.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5937.png,Normal/masks/Normal-5937.png,True,False,299,299,L,104.78,cac48c5a22d7189bcf1e7c6a7016cc7e
Normal-5938.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5938.png,Normal/images/Normal-5938.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5938.png,Normal/masks/Normal-5938.png,True,False,299,299,L,138.08,04f59da76c14f52213720664c70d14f6
Normal-5939.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5939.png,Normal/images/Normal-5939.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5939.png,Normal/masks/Normal-5939.png,True,False,299,299,L,102.2,78508ec4204c38196f955edd1afb43a3
Normal-594.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-594.png,Normal/images/Normal-594.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-594.png,Normal/masks/Normal-594.png,True,False,299,299,L,130.18,a9277a55382b95797c7d1702e4cc8fe2
Normal-5940.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5940.png,Normal/images/Normal-5940.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5940.png,Normal/masks/Normal-5940.png,True,False,299,299,L,119.15,f731dab7ccd40a119c8ba09eb1049d1f
Normal-5941.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5941.png,Normal/images/Normal-5941.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5941.png,Normal/masks/Normal-5941.png,True,False,299,299,L,137.52,464c1ac4a7382dce58aabc28365b3292
Normal-5942.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5942.png,Normal/images/Normal-5942.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5942.png,Normal/masks/Normal-5942.png,True,False,299,299,L,128.0,7914659619dc520ff51bf551557f3686
Normal-5943.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5943.png,Normal/images/Normal-5943.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5943.png,Normal/masks/Normal-5943.png,True,False,299,299,L,124.42,ada3748067dc820b955f00a5662cecad
Normal-5944.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5944.png,Normal/images/Normal-5944.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5944.png,Normal/masks/Normal-5944.png,True,False,299,299,L,125.95,90d7dabdb7815ed93b325e8c9082c289
Normal-5945.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5945.png,Normal/images/Normal-5945.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5945.png,Normal/masks/Normal-5945.png,True,False,299,299,L,155.55,e56b8da7eece7aa753084ef754589250
Normal-5946.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5946.png,Normal/images/Normal-5946.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5946.png,Normal/masks/Normal-5946.png,True,False,299,299,L,129.81,3829cbf2a447ed21121e3019138a2faf
Normal-5947.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5947.png,Normal/images/Normal-5947.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5947.png,Normal/masks/Normal-5947.png,True,False,299,299,L,113.87,622fae73029a0750bb3f1aeb4b75632f
Normal-5948.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5948.png,Normal/images/Normal-5948.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5948.png,Normal/masks/Normal-5948.png,True,False,299,299,L,105.69,4e0cf558122619826dc08494992823eb
Normal-5949.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5949.png,Normal/images/Normal-5949.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5949.png,Normal/masks/Normal-5949.png,True,False,299,299,L,130.73,a3ef3f6743b2147ca77b00bd00d902ba
Normal-595.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-595.png,Normal/images/Normal-595.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-595.png,Normal/masks/Normal-595.png,True,False,299,299,L,127.63,7d99af96607694ab67a54c327482bc68
Normal-5950.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5950.png,Normal/images/Normal-5950.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5950.png,Normal/masks/Normal-5950.png,True,False,299,299,L,157.76,26a499f90470e79abf65c5b3be51eff2
Normal-5951.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5951.png,Normal/images/Normal-5951.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5951.png,Normal/masks/Normal-5951.png,True,False,299,299,L,158.17,9b7b38152a5b55a14bd72640c8153350
Normal-5952.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5952.png,Normal/images/Normal-5952.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5952.png,Normal/masks/Normal-5952.png,True,False,299,299,L,106.57,29e84522ad03ee8a82cd863564408d76
Normal-5953.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5953.png,Normal/images/Normal-5953.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5953.png,Normal/masks/Normal-5953.png,True,False,299,299,L,124.88,274752723f4bfb12c8403ec69d27dc50
Normal-5954.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5954.png,Normal/images/Normal-5954.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5954.png,Normal/masks/Normal-5954.png,True,False,299,299,L,124.03,78a477795b674cb5a3b63b67344b967f
Normal-5955.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5955.png,Normal/images/Normal-5955.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5955.png,Normal/masks/Normal-5955.png,True,False,299,299,L,131.65,ca7329d96d8cf76fd76fe78242d4345e
Normal-5956.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5956.png,Normal/images/Normal-5956.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5956.png,Normal/masks/Normal-5956.png,True,False,299,299,L,115.48,2c75943934c56bd42425721d2d260444
Normal-5957.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5957.png,Normal/images/Normal-5957.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5957.png,Normal/masks/Normal-5957.png,True,False,299,299,L,152.5,ff09040e61864f7579a407041581ca38
Normal-5958.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5958.png,Normal/images/Normal-5958.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5958.png,Normal/masks/Normal-5958.png,True,False,299,299,L,171.82,3fb4a0881a6ef4c52b8e3f314a3a9d4c
Normal-5959.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5959.png,Normal/images/Normal-5959.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5959.png,Normal/masks/Normal-5959.png,True,False,299,299,L,118.36,0263e65d0f65399864bb77d1b6132c8f
Normal-596.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-596.png,Normal/images/Normal-596.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-596.png,Normal/masks/Normal-596.png,True,False,299,299,L,118.79,5a4a91712a4ebf0fc6894d14e6b3c00b
Normal-5960.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5960.png,Normal/images/Normal-5960.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5960.png,Normal/masks/Normal-5960.png,True,False,299,299,L,107.55,bc810ec62bc9d7954052832554dbd76b
Normal-5961.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5961.png,Normal/images/Normal-5961.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5961.png,Normal/masks/Normal-5961.png,True,False,299,299,L,121.68,042870a44c737d90dee2fbecdc1c784f
Normal-5962.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5962.png,Normal/images/Normal-5962.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5962.png,Normal/masks/Normal-5962.png,True,False,299,299,L,166.92,bbaf6d7f0e163be39c79d024b7d7e572
Normal-5963.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5963.png,Normal/images/Normal-5963.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5963.png,Normal/masks/Normal-5963.png,True,False,299,299,L,112.92,3560ac6827d2070c54826d7e09ccdfe8
Normal-5964.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5964.png,Normal/images/Normal-5964.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5964.png,Normal/masks/Normal-5964.png,True,False,299,299,L,124.73,b9ec457dfe7c191664a6c3fb19695c84
Normal-5965.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5965.png,Normal/images/Normal-5965.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5965.png,Normal/masks/Normal-5965.png,True,False,299,299,L,142.04,1fc7880fa3daf65218fed0f9484d76ed
Normal-5966.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5966.png,Normal/images/Normal-5966.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5966.png,Normal/masks/Normal-5966.png,True,False,299,299,L,129.53,74b7bac2b4babb381e50bd88f5d31db9
Normal-5967.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5967.png,Normal/images/Normal-5967.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5967.png,Normal/masks/Normal-5967.png,True,False,299,299,L,120.75,654f6f541e68b33b369ee098d678b3bb
Normal-5968.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5968.png,Normal/images/Normal-5968.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5968.png,Normal/masks/Normal-5968.png,True,False,299,299,L,116.71,e1ed381a66e7ce2463909523d99568dd
Normal-5969.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5969.png,Normal/images/Normal-5969.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5969.png,Normal/masks/Normal-5969.png,True,False,299,299,L,139.03,ee29599f38ce341543b34f4b4c16f067
Normal-597.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-597.png,Normal/images/Normal-597.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-597.png,Normal/masks/Normal-597.png,True,False,299,299,L,95.23,bea653dafc2ca225a5a38579bf9cbff6
Normal-5970.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5970.png,Normal/images/Normal-5970.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5970.png,Normal/masks/Normal-5970.png,True,False,299,299,L,130.17,21ee01cbf01046e2a419ba27c503e0cc
Normal-5971.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5971.png,Normal/images/Normal-5971.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5971.png,Normal/masks/Normal-5971.png,True,False,299,299,L,112.28,7c840501822ab2fbab8baec9f739678a
Normal-5972.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5972.png,Normal/images/Normal-5972.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5972.png,Normal/masks/Normal-5972.png,True,False,299,299,L,122.91,cbd08774bfc59a2b102778554b376582
Normal-5973.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5973.png,Normal/images/Normal-5973.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5973.png,Normal/masks/Normal-5973.png,True,False,299,299,L,136.04,1962207a5b2ef219eac0099631031ed6
Normal-5974.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5974.png,Normal/images/Normal-5974.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5974.png,Normal/masks/Normal-5974.png,True,False,299,299,L,163.64,ae20e21a55555d94c6bb059babd27296
Normal-5975.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5975.png,Normal/images/Normal-5975.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5975.png,Normal/masks/Normal-5975.png,True,False,299,299,L,107.58,decee1cf761f44791d9cfddce29eeb74
Normal-5976.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5976.png,Normal/images/Normal-5976.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5976.png,Normal/masks/Normal-5976.png,True,False,299,299,L,135.05,edfa2de8f289dd161b80b0fb750ef1d8
Normal-5977.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5977.png,Normal/images/Normal-5977.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5977.png,Normal/masks/Normal-5977.png,True,False,299,299,L,105.07,c2f3dfa250ec4c88891d381371217cdb
Normal-5978.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5978.png,Normal/images/Normal-5978.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5978.png,Normal/masks/Normal-5978.png,True,False,299,299,L,102.59,3db213bc31d7bd4334a559fde971e61e
Normal-5979.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5979.png,Normal/images/Normal-5979.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5979.png,Normal/masks/Normal-5979.png,True,False,299,299,L,149.92,f91d0a07f03af270e5db337ae9b0f7db
Normal-598.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-598.png,Normal/images/Normal-598.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-598.png,Normal/masks/Normal-598.png,True,False,299,299,L,112.0,daa231beb279585614c1b7ec7c25cd5a
Normal-5980.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5980.png,Normal/images/Normal-5980.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5980.png,Normal/masks/Normal-5980.png,True,False,299,299,L,94.39,fc3ed994e0c49a2b3933f7e7e049b9b1
Normal-5981.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5981.png,Normal/images/Normal-5981.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5981.png,Normal/masks/Normal-5981.png,True,False,299,299,L,131.15,94604ba36a5eda6cb1daa0aec7f74734
Normal-5982.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5982.png,Normal/images/Normal-5982.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5982.png,Normal/masks/Normal-5982.png,True,False,299,299,L,133.26,c61c41314ec7591d345c248dd95e4cf2
Normal-5983.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5983.png,Normal/images/Normal-5983.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5983.png,Normal/masks/Normal-5983.png,True,False,299,299,L,110.17,71c6e1507532661db2dcc5f117bb748b
Normal-5984.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5984.png,Normal/images/Normal-5984.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5984.png,Normal/masks/Normal-5984.png,True,False,299,299,L,113.63,66454cebe2a965c73db50e85cc80d079
Normal-5985.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5985.png,Normal/images/Normal-5985.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5985.png,Normal/masks/Normal-5985.png,True,False,299,299,L,124.28,7dc930e39aa3e66743a84b5ece1a2329
Normal-5986.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5986.png,Normal/images/Normal-5986.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5986.png,Normal/masks/Normal-5986.png,True,False,299,299,L,114.86,ec7735198bc41aecf91364e92159925c
Normal-5987.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5987.png,Normal/images/Normal-5987.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5987.png,Normal/masks/Normal-5987.png,True,False,299,299,L,170.12,ec2f5a923668db5090d180d6a0fa41d9
Normal-5988.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5988.png,Normal/images/Normal-5988.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5988.png,Normal/masks/Normal-5988.png,True,False,299,299,L,165.0,87443ca6d508e737964cd586d6ef17d8
Normal-5989.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5989.png,Normal/images/Normal-5989.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5989.png,Normal/masks/Normal-5989.png,True,False,299,299,L,135.61,ce0f7897e6e2080db38dd74c43bb401f
Normal-599.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-599.png,Normal/images/Normal-599.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-599.png,Normal/masks/Normal-599.png,True,False,299,299,L,127.7,c958533b5e18ec33e951b234538ceda0
Normal-5990.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5990.png,Normal/images/Normal-5990.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5990.png,Normal/masks/Normal-5990.png,True,False,299,299,L,164.66,2d76a8c61fe2e4a545f19d501df57cbe
Normal-5991.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5991.png,Normal/images/Normal-5991.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5991.png,Normal/masks/Normal-5991.png,True,False,299,299,L,115.01,2e933eb46042cbbc187c48deb925b247
Normal-5992.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5992.png,Normal/images/Normal-5992.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5992.png,Normal/masks/Normal-5992.png,True,False,299,299,L,103.02,8532e274833b3cd36e6f697a42a74e6a
Normal-5993.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5993.png,Normal/images/Normal-5993.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5993.png,Normal/masks/Normal-5993.png,True,False,299,299,L,111.29,205d3d26f212c535e64fb0e3b261af46
Normal-5994.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5994.png,Normal/images/Normal-5994.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5994.png,Normal/masks/Normal-5994.png,True,False,299,299,L,125.15,b98da6e050f6bd6b53d2f0ced0ae58e7
Normal-5995.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5995.png,Normal/images/Normal-5995.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5995.png,Normal/masks/Normal-5995.png,True,False,299,299,L,144.74,ef8215fb2cb1d54f3dcaeec8d668892f
Normal-5996.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5996.png,Normal/images/Normal-5996.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5996.png,Normal/masks/Normal-5996.png,True,False,299,299,L,119.86,6e56faf99ca3bcacd9076320c5c803bf
Normal-5997.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5997.png,Normal/images/Normal-5997.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5997.png,Normal/masks/Normal-5997.png,True,False,299,299,L,89.83,caefaaee052dfe3ea4a6a6c0af0e43d3
Normal-5998.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5998.png,Normal/images/Normal-5998.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5998.png,Normal/masks/Normal-5998.png,True,False,299,299,L,134.76,09a3397018db795d4bd90f02a902c149
Normal-5999.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-5999.png,Normal/images/Normal-5999.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-5999.png,Normal/masks/Normal-5999.png,True,False,299,299,L,100.26,0343bbce6925b7496304aac10328b817
Normal-6.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6.png,Normal/images/Normal-6.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6.png,Normal/masks/Normal-6.png,True,False,299,299,L,111.95,c1ccb40667c7d63a2119f65616dffe5c
Normal-60.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-60.png,Normal/images/Normal-60.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-60.png,Normal/masks/Normal-60.png,True,False,299,299,L,117.54,cde4dd7e5f48c72842dea260381d11d1
Normal-600.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-600.png,Normal/images/Normal-600.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-600.png,Normal/masks/Normal-600.png,True,False,299,299,L,114.07,4380f317a31066b1cd01ffc129e0e2f4
Normal-6000.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6000.png,Normal/images/Normal-6000.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6000.png,Normal/masks/Normal-6000.png,True,False,299,299,L,112.94,b419f220eedc588fcd0ab853882abd34
Normal-6001.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6001.png,Normal/images/Normal-6001.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6001.png,Normal/masks/Normal-6001.png,True,False,299,299,L,109.53,77712412da1ee40690d5231cc0dd4dcd
Normal-6002.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6002.png,Normal/images/Normal-6002.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6002.png,Normal/masks/Normal-6002.png,True,False,299,299,L,120.68,1076304abd61115f921c6e313e0100e5
Normal-6003.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6003.png,Normal/images/Normal-6003.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6003.png,Normal/masks/Normal-6003.png,True,False,299,299,L,136.91,5fc6c2055379d86420b74345e2f78c6b
Normal-6004.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6004.png,Normal/images/Normal-6004.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6004.png,Normal/masks/Normal-6004.png,True,False,299,299,L,111.22,79287ac839372174ca7df243449249b9
Normal-6005.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6005.png,Normal/images/Normal-6005.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6005.png,Normal/masks/Normal-6005.png,True,False,299,299,L,140.01,18283f8164c63099e20b1423747dda31
Normal-6006.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6006.png,Normal/images/Normal-6006.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6006.png,Normal/masks/Normal-6006.png,True,False,299,299,L,124.84,1c012b557970e92ed2036f836c005fdb
Normal-6007.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6007.png,Normal/images/Normal-6007.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6007.png,Normal/masks/Normal-6007.png,True,False,299,299,L,114.02,424d5c6bbf877f17a6d1a203ce6bd441
Normal-6008.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6008.png,Normal/images/Normal-6008.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6008.png,Normal/masks/Normal-6008.png,True,False,299,299,L,97.38,dcd1903e875c00c4947703f63d537632
Normal-6009.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6009.png,Normal/images/Normal-6009.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6009.png,Normal/masks/Normal-6009.png,True,False,299,299,L,107.68,7240f6ebcd7efc9559dbe30faf34a501
Normal-601.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-601.png,Normal/images/Normal-601.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-601.png,Normal/masks/Normal-601.png,True,False,299,299,L,117.62,7c57c0b00c83f912ddf8e5342f76cb92
Normal-6010.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6010.png,Normal/images/Normal-6010.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6010.png,Normal/masks/Normal-6010.png,True,False,299,299,L,119.5,5d02e03fcd7cbe9e6e4e248c5161e630
Normal-6011.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6011.png,Normal/images/Normal-6011.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6011.png,Normal/masks/Normal-6011.png,True,False,299,299,L,152.98,5b2faa3dc95273fad2f96b0232e11953
Normal-6012.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6012.png,Normal/images/Normal-6012.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6012.png,Normal/masks/Normal-6012.png,True,False,299,299,L,159.88,d19d8d3e0bf6358274c601a8ef327969
Normal-6013.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6013.png,Normal/images/Normal-6013.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6013.png,Normal/masks/Normal-6013.png,True,False,299,299,L,164.08,9630117326e7ef3bbbd300c7c04ede03
Normal-6014.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6014.png,Normal/images/Normal-6014.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6014.png,Normal/masks/Normal-6014.png,True,False,299,299,L,166.23,a8f195f79e17faca4d627ad936e65e1f
Normal-6015.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6015.png,Normal/images/Normal-6015.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6015.png,Normal/masks/Normal-6015.png,True,False,299,299,L,134.54,69bea1e4672470786ceeccf692239f62
Normal-6016.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6016.png,Normal/images/Normal-6016.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6016.png,Normal/masks/Normal-6016.png,True,False,299,299,L,126.29,e1be09b2ee9ea7fcaa49503f398c9fac
Normal-6017.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6017.png,Normal/images/Normal-6017.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6017.png,Normal/masks/Normal-6017.png,True,False,299,299,L,125.58,04b8efeaafa370fca0a5d5278d7fde71
Normal-6018.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6018.png,Normal/images/Normal-6018.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6018.png,Normal/masks/Normal-6018.png,True,False,299,299,L,175.85,c34cffed6d4b530f0e94076af94cd6fc
Normal-6019.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6019.png,Normal/images/Normal-6019.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6019.png,Normal/masks/Normal-6019.png,True,False,299,299,L,103.22,3bd8916969e6b03d0a5c5e895ba700b8
Normal-602.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-602.png,Normal/images/Normal-602.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-602.png,Normal/masks/Normal-602.png,True,False,299,299,L,126.12,315dd76e7ff101a6e8410afa40c46303
Normal-6020.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6020.png,Normal/images/Normal-6020.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6020.png,Normal/masks/Normal-6020.png,True,False,299,299,L,115.09,03a76295365826d270cf1cfd30eb1971
Normal-6021.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6021.png,Normal/images/Normal-6021.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6021.png,Normal/masks/Normal-6021.png,True,False,299,299,L,114.8,3f2aaad7a53fc3f034da5502f760cc25
Normal-6022.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6022.png,Normal/images/Normal-6022.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6022.png,Normal/masks/Normal-6022.png,True,False,299,299,L,110.83,921d26b1a64eadfb427f5e5a8c95f738
Normal-6023.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6023.png,Normal/images/Normal-6023.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6023.png,Normal/masks/Normal-6023.png,True,False,299,299,L,94.63,d4fc2cf2eb8546ae15d64b2f48234d94
Normal-6024.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6024.png,Normal/images/Normal-6024.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6024.png,Normal/masks/Normal-6024.png,True,False,299,299,L,112.35,e14a06d873ac37a0c3f47ac04b719f6c
Normal-6025.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6025.png,Normal/images/Normal-6025.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6025.png,Normal/masks/Normal-6025.png,True,False,299,299,L,158.72,f03cec14e1d219e4802684926d544001
Normal-6026.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6026.png,Normal/images/Normal-6026.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6026.png,Normal/masks/Normal-6026.png,True,False,299,299,L,128.54,754ea5476ad68e7d1e422254d8d5f3bd
Normal-6027.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6027.png,Normal/images/Normal-6027.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6027.png,Normal/masks/Normal-6027.png,True,False,299,299,L,98.19,f0d219512237677b3d68f2609b5c0038
Normal-6028.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6028.png,Normal/images/Normal-6028.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6028.png,Normal/masks/Normal-6028.png,True,False,299,299,L,160.25,26feeeabd5bbabb5ee6158aaaee3de9c
Normal-6029.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6029.png,Normal/images/Normal-6029.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6029.png,Normal/masks/Normal-6029.png,True,False,299,299,L,124.53,a8864fddf392aee938c3b3f781b8650c
Normal-603.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-603.png,Normal/images/Normal-603.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-603.png,Normal/masks/Normal-603.png,True,False,299,299,L,128.35,118b8b1bdfacb47e598b978d470ca81e
Normal-6030.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6030.png,Normal/images/Normal-6030.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6030.png,Normal/masks/Normal-6030.png,True,False,299,299,L,148.93,2cb3142cee3f5aa320081e4b63551a59
Normal-6031.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6031.png,Normal/images/Normal-6031.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6031.png,Normal/masks/Normal-6031.png,True,False,299,299,L,128.4,abe8c3417133e4beaa8c9764d5a919a0
Normal-6032.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6032.png,Normal/images/Normal-6032.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6032.png,Normal/masks/Normal-6032.png,True,False,299,299,L,145.99,65cf8d657cf4a202a09b03c0ecb61a50
Normal-6033.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6033.png,Normal/images/Normal-6033.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6033.png,Normal/masks/Normal-6033.png,True,False,299,299,L,104.21,01112681508260dbc83de1c7098857be
Normal-6034.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6034.png,Normal/images/Normal-6034.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6034.png,Normal/masks/Normal-6034.png,True,False,299,299,L,103.53,10e531553b49c08c3a17e329d448cb15
Normal-6035.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6035.png,Normal/images/Normal-6035.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6035.png,Normal/masks/Normal-6035.png,True,False,299,299,L,158.63,4ac49fea107ad85df0a8be85bf46f3a1
Normal-6036.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6036.png,Normal/images/Normal-6036.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6036.png,Normal/masks/Normal-6036.png,True,False,299,299,L,127.09,97b90a114a3ecf9f4ea343182e161fed
Normal-6037.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6037.png,Normal/images/Normal-6037.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6037.png,Normal/masks/Normal-6037.png,True,False,299,299,L,136.73,73d2432ede5ee5ecff5740ab90c0e589
Normal-6038.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6038.png,Normal/images/Normal-6038.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6038.png,Normal/masks/Normal-6038.png,True,False,299,299,L,184.47,317fe74d996f6ffa7711362bfb1e5780
Normal-6039.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6039.png,Normal/images/Normal-6039.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6039.png,Normal/masks/Normal-6039.png,True,False,299,299,L,161.92,d20cce99e22e65515b14ad4adfff4f8d
Normal-604.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-604.png,Normal/images/Normal-604.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-604.png,Normal/masks/Normal-604.png,True,False,299,299,L,115.1,92260015e80a838fdbf841ba7745263a
Normal-6040.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6040.png,Normal/images/Normal-6040.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6040.png,Normal/masks/Normal-6040.png,True,False,299,299,L,152.43,90a74c0cc3adf80aea40ba97bc8cf271
Normal-6041.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6041.png,Normal/images/Normal-6041.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6041.png,Normal/masks/Normal-6041.png,True,False,299,299,L,154.66,5dd2c9279a76ab1c3c142d99774ce0eb
Normal-6042.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6042.png,Normal/images/Normal-6042.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6042.png,Normal/masks/Normal-6042.png,True,False,299,299,L,128.51,e0fb8006968d214ea4e377ff21ec8edf
Normal-6043.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6043.png,Normal/images/Normal-6043.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6043.png,Normal/masks/Normal-6043.png,True,False,299,299,L,113.38,44a9a300911ad858af41d88eea2e4717
Normal-6044.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6044.png,Normal/images/Normal-6044.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6044.png,Normal/masks/Normal-6044.png,True,False,299,299,L,128.54,f78170f88bc7bdf6b2cb49cb897f414b
Normal-6045.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6045.png,Normal/images/Normal-6045.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6045.png,Normal/masks/Normal-6045.png,True,False,299,299,L,78.1,2ff410d944c64fe284427a370d9dbb98
Normal-6046.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6046.png,Normal/images/Normal-6046.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6046.png,Normal/masks/Normal-6046.png,True,False,299,299,L,116.68,d7c44bd1726b3415a5b0ac47d3731b21
Normal-6047.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6047.png,Normal/images/Normal-6047.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6047.png,Normal/masks/Normal-6047.png,True,False,299,299,L,156.3,eaffe3b793b4028ef46b48e0697a3ccc
Normal-6048.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6048.png,Normal/images/Normal-6048.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6048.png,Normal/masks/Normal-6048.png,True,False,299,299,L,169.15,d7c394774f5777ab35b82c827d5ef515
Normal-6049.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6049.png,Normal/images/Normal-6049.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6049.png,Normal/masks/Normal-6049.png,True,False,299,299,L,150.11,35d40eb3f51c944bc959c2b9048cd18d
Normal-605.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-605.png,Normal/images/Normal-605.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-605.png,Normal/masks/Normal-605.png,True,False,299,299,L,121.85,423158220199d4ae4f1331508b3b9098
Normal-6050.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6050.png,Normal/images/Normal-6050.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6050.png,Normal/masks/Normal-6050.png,True,False,299,299,L,104.15,9d495867f794df91e38cc314e2cd1f0e
Normal-6051.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6051.png,Normal/images/Normal-6051.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6051.png,Normal/masks/Normal-6051.png,True,False,299,299,L,165.21,899764dbca37c9c2b1b0ea39ab1c9818
Normal-6052.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6052.png,Normal/images/Normal-6052.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6052.png,Normal/masks/Normal-6052.png,True,False,299,299,L,131.92,0fdaacc6dd0cc154123bbe9104481fd2
Normal-6053.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6053.png,Normal/images/Normal-6053.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6053.png,Normal/masks/Normal-6053.png,True,False,299,299,L,160.45,0330276e635e6d876de531178fa0af33
Normal-6054.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6054.png,Normal/images/Normal-6054.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6054.png,Normal/masks/Normal-6054.png,True,False,299,299,L,113.77,ba4caeddd348c6b69c97b8eb6628a24c
Normal-6055.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6055.png,Normal/images/Normal-6055.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6055.png,Normal/masks/Normal-6055.png,True,False,299,299,L,120.2,2f7ff4be53ad0b35a5fafe95dd73012a
Normal-6056.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6056.png,Normal/images/Normal-6056.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6056.png,Normal/masks/Normal-6056.png,True,False,299,299,L,110.66,001342b3c95d30c9e4aa88bc35ba32a3
Normal-6057.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6057.png,Normal/images/Normal-6057.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6057.png,Normal/masks/Normal-6057.png,True,False,299,299,L,108.18,0760d7c8d503379a524829b371922512
Normal-6058.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6058.png,Normal/images/Normal-6058.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6058.png,Normal/masks/Normal-6058.png,True,False,299,299,L,167.4,5513c7ab3c4c08a8668464cd3a3ae5f7
Normal-6059.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6059.png,Normal/images/Normal-6059.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6059.png,Normal/masks/Normal-6059.png,True,False,299,299,L,97.67,827c960eceadde0fd7639166b39c5528
Normal-606.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-606.png,Normal/images/Normal-606.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-606.png,Normal/masks/Normal-606.png,True,False,299,299,L,118.28,a8ba2b98a2b53554e5f999dcb19791f0
Normal-6060.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6060.png,Normal/images/Normal-6060.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6060.png,Normal/masks/Normal-6060.png,True,False,299,299,L,118.08,16a5d2f008a89b7f705a5dba2e1e7f80
Normal-6061.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6061.png,Normal/images/Normal-6061.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6061.png,Normal/masks/Normal-6061.png,True,False,299,299,L,141.53,cdf4a3b9ed29de5f68aa5efbfb22bf39
Normal-6062.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6062.png,Normal/images/Normal-6062.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6062.png,Normal/masks/Normal-6062.png,True,False,299,299,L,161.48,d48b6005951056d011cbdd2dcce043f4
Normal-6063.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6063.png,Normal/images/Normal-6063.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6063.png,Normal/masks/Normal-6063.png,True,False,299,299,L,156.87,7ac43bfa682f82acac162788966f56d0
Normal-6064.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6064.png,Normal/images/Normal-6064.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6064.png,Normal/masks/Normal-6064.png,True,False,299,299,L,124.24,657eaf47d061fa30ecc0998975b4f8e3
Normal-6065.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6065.png,Normal/images/Normal-6065.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6065.png,Normal/masks/Normal-6065.png,True,False,299,299,L,120.18,298f5999fa4e40b2c022dfccf5d30e92
Normal-6066.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6066.png,Normal/images/Normal-6066.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6066.png,Normal/masks/Normal-6066.png,True,False,299,299,L,140.34,1fd875ef523cee3d717b0dbebdcf816b
Normal-6067.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6067.png,Normal/images/Normal-6067.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6067.png,Normal/masks/Normal-6067.png,True,False,299,299,L,107.39,094b7d8117e98469dc0148d27ac797eb
Normal-6068.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6068.png,Normal/images/Normal-6068.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6068.png,Normal/masks/Normal-6068.png,True,False,299,299,L,107.85,3bf0fc9c74c5b393d7e5c9af602cb6d5
Normal-6069.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6069.png,Normal/images/Normal-6069.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6069.png,Normal/masks/Normal-6069.png,True,False,299,299,L,130.52,ba5f2d4a05d308d423f0291f2d927c3f
Normal-607.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-607.png,Normal/images/Normal-607.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-607.png,Normal/masks/Normal-607.png,True,False,299,299,L,104.52,9d200aa1c72760913c78bbc26f65d489
Normal-6070.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6070.png,Normal/images/Normal-6070.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6070.png,Normal/masks/Normal-6070.png,True,False,299,299,L,116.16,740ffd891ff353cf24142cc77076cd9b
Normal-6071.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6071.png,Normal/images/Normal-6071.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6071.png,Normal/masks/Normal-6071.png,True,False,299,299,L,182.12,dec7b255c5129213f334bcab2a42d348
Normal-6072.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6072.png,Normal/images/Normal-6072.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6072.png,Normal/masks/Normal-6072.png,True,False,299,299,L,138.73,074dfcd5c9c64fae857827e5eeff3008
Normal-6073.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6073.png,Normal/images/Normal-6073.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6073.png,Normal/masks/Normal-6073.png,True,False,299,299,L,130.58,0ee3497dd12d0bbfad56071903bdf956
Normal-6074.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6074.png,Normal/images/Normal-6074.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6074.png,Normal/masks/Normal-6074.png,True,False,299,299,L,124.31,c69dfac41315bc81282550895bdf2013
Normal-6075.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6075.png,Normal/images/Normal-6075.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6075.png,Normal/masks/Normal-6075.png,True,False,299,299,L,109.44,02f7f15847df6879bf295e65eb046c2e
Normal-6076.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6076.png,Normal/images/Normal-6076.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6076.png,Normal/masks/Normal-6076.png,True,False,299,299,L,161.8,8f3aab5a6367225f47348e1b425e96be
Normal-6077.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6077.png,Normal/images/Normal-6077.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6077.png,Normal/masks/Normal-6077.png,True,False,299,299,L,128.78,94518ee99542b9874b53178c03c68db8
Normal-6078.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6078.png,Normal/images/Normal-6078.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6078.png,Normal/masks/Normal-6078.png,True,False,299,299,L,110.83,5ea399489c171ea84b3f888c470195d4
Normal-6079.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6079.png,Normal/images/Normal-6079.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6079.png,Normal/masks/Normal-6079.png,True,False,299,299,L,169.11,aca15e52f65e48d2be13babc90958399
Normal-608.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-608.png,Normal/images/Normal-608.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-608.png,Normal/masks/Normal-608.png,True,False,299,299,L,112.86,609dfca3ae6625c4881fc012c9c6f3f2
Normal-6080.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6080.png,Normal/images/Normal-6080.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6080.png,Normal/masks/Normal-6080.png,True,False,299,299,L,117.17,6b43087c3432a42f47c9d0787b07e3bd
Normal-6081.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6081.png,Normal/images/Normal-6081.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6081.png,Normal/masks/Normal-6081.png,True,False,299,299,L,117.66,209b72af6c3a63c462c77846e6275c68
Normal-6082.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6082.png,Normal/images/Normal-6082.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6082.png,Normal/masks/Normal-6082.png,True,False,299,299,L,102.05,323e5b764e6d2e7940118eaffc6831b7
Normal-6083.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6083.png,Normal/images/Normal-6083.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6083.png,Normal/masks/Normal-6083.png,True,False,299,299,L,134.68,9c1eeb60df6eeabe23137901cec45093
Normal-6084.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6084.png,Normal/images/Normal-6084.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6084.png,Normal/masks/Normal-6084.png,True,False,299,299,L,97.97,c004b017650da65e1572fdaeafe50ef6
Normal-6085.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6085.png,Normal/images/Normal-6085.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6085.png,Normal/masks/Normal-6085.png,True,False,299,299,L,135.04,f17f59e820499b0604701f0d34df1e93
Normal-6086.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6086.png,Normal/images/Normal-6086.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6086.png,Normal/masks/Normal-6086.png,True,False,299,299,L,142.46,b31195508036f31ba856a58bc795ebf4
Normal-6087.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6087.png,Normal/images/Normal-6087.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6087.png,Normal/masks/Normal-6087.png,True,False,299,299,L,109.62,3757eea8225673147d45cbf90744b80a
Normal-6088.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6088.png,Normal/images/Normal-6088.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6088.png,Normal/masks/Normal-6088.png,True,False,299,299,L,97.4,5fdb91977caaed5eb62dce4ff8cd834f
Normal-6089.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6089.png,Normal/images/Normal-6089.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6089.png,Normal/masks/Normal-6089.png,True,False,299,299,L,140.29,ba70002ec0075410bf5c76a76b733e56
Normal-609.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-609.png,Normal/images/Normal-609.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-609.png,Normal/masks/Normal-609.png,True,False,299,299,L,119.29,a6fd442e6a553f05333eb6c440788fa9
Normal-6090.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6090.png,Normal/images/Normal-6090.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6090.png,Normal/masks/Normal-6090.png,True,False,299,299,L,100.56,9ab76f82d11ef6988a0a7d41cbda57f3
Normal-6091.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6091.png,Normal/images/Normal-6091.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6091.png,Normal/masks/Normal-6091.png,True,False,299,299,L,158.51,49c545551751d501ae3b47ca2ce71a2b
Normal-6092.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6092.png,Normal/images/Normal-6092.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6092.png,Normal/masks/Normal-6092.png,True,False,299,299,L,123.59,fb66124ba8f4d281c6e9e9edf69d1f0b
Normal-6093.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6093.png,Normal/images/Normal-6093.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6093.png,Normal/masks/Normal-6093.png,True,False,299,299,L,117.26,752ccb2afff0d2cc9f324fe3ef37855a
Normal-6094.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6094.png,Normal/images/Normal-6094.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6094.png,Normal/masks/Normal-6094.png,True,False,299,299,L,121.25,43df5de6bc22793fc5c54c4897e5feb6
Normal-6095.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6095.png,Normal/images/Normal-6095.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6095.png,Normal/masks/Normal-6095.png,True,False,299,299,L,129.49,f1a6bcc4546acefc4292aa4205041e20
Normal-6096.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6096.png,Normal/images/Normal-6096.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6096.png,Normal/masks/Normal-6096.png,True,False,299,299,L,110.7,2aa665537941fb75c76287c2d898ad03
Normal-6097.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6097.png,Normal/images/Normal-6097.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6097.png,Normal/masks/Normal-6097.png,True,False,299,299,L,133.14,5ef536c0d2a32af1c3cd07b528781892
Normal-6098.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6098.png,Normal/images/Normal-6098.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6098.png,Normal/masks/Normal-6098.png,True,False,299,299,L,152.75,b3dd7c6d09b56f64056c8ab6264f71ca
Normal-6099.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6099.png,Normal/images/Normal-6099.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6099.png,Normal/masks/Normal-6099.png,True,False,299,299,L,173.47,a914fd2b7d1f95cbedf4a4d36cd4c217
Normal-61.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-61.png,Normal/images/Normal-61.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-61.png,Normal/masks/Normal-61.png,True,False,299,299,L,123.87,f3893bb6642a7bac7a2e516cddfa6868
Normal-610.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-610.png,Normal/images/Normal-610.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-610.png,Normal/masks/Normal-610.png,True,False,299,299,L,111.73,eb29823a3099f35824db3556b594e11f
Normal-6100.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6100.png,Normal/images/Normal-6100.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6100.png,Normal/masks/Normal-6100.png,True,False,299,299,L,116.43,42ef96c6bff09f27404de85da2ea45de
Normal-6101.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6101.png,Normal/images/Normal-6101.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6101.png,Normal/masks/Normal-6101.png,True,False,299,299,L,112.1,080564b453e123e639b54262700cfa07
Normal-6102.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6102.png,Normal/images/Normal-6102.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6102.png,Normal/masks/Normal-6102.png,True,False,299,299,L,177.39,e8a785d26cdc91084477310c1a64d50d
Normal-6103.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6103.png,Normal/images/Normal-6103.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6103.png,Normal/masks/Normal-6103.png,True,False,299,299,L,123.83,cea0e9e46557ff17cd22ff6fa8ec14fd
Normal-6104.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6104.png,Normal/images/Normal-6104.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6104.png,Normal/masks/Normal-6104.png,True,False,299,299,L,166.78,8b2689cb8b19f4f4317f123557fcae29
Normal-6105.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6105.png,Normal/images/Normal-6105.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6105.png,Normal/masks/Normal-6105.png,True,False,299,299,L,102.47,fddc1fe3b10ecc27340f6794e81fa054
Normal-6106.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6106.png,Normal/images/Normal-6106.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6106.png,Normal/masks/Normal-6106.png,True,False,299,299,L,137.35,b4235e5ad47616a279162d94ac4283d0
Normal-6107.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6107.png,Normal/images/Normal-6107.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6107.png,Normal/masks/Normal-6107.png,True,False,299,299,L,130.39,3d0ed7722fc66638af2453be66b853c1
Normal-6108.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6108.png,Normal/images/Normal-6108.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6108.png,Normal/masks/Normal-6108.png,True,False,299,299,L,148.9,fe130ef0ec1401114c0ad217fda56980
Normal-6109.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6109.png,Normal/images/Normal-6109.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6109.png,Normal/masks/Normal-6109.png,True,False,299,299,L,117.75,3540f691599eb4fd2cb771ea56476d1d
Normal-611.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-611.png,Normal/images/Normal-611.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-611.png,Normal/masks/Normal-611.png,True,False,299,299,L,108.33,8b1b9390d2539065138c3c90477cc28d
Normal-6110.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6110.png,Normal/images/Normal-6110.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6110.png,Normal/masks/Normal-6110.png,True,False,299,299,L,131.68,f33dcf1cb9764f9d1c43f1b983a466d4
Normal-6111.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6111.png,Normal/images/Normal-6111.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6111.png,Normal/masks/Normal-6111.png,True,False,299,299,L,166.72,c0dbf1d9b18d0ab83d572639a5dd512e
Normal-6112.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6112.png,Normal/images/Normal-6112.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6112.png,Normal/masks/Normal-6112.png,True,False,299,299,L,144.65,75e4189affee48ca4652999a5ae4f24a
Normal-6113.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6113.png,Normal/images/Normal-6113.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6113.png,Normal/masks/Normal-6113.png,True,False,299,299,L,137.32,76f8dcd95e292878985adcdd44aba7b2
Normal-6114.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6114.png,Normal/images/Normal-6114.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6114.png,Normal/masks/Normal-6114.png,True,False,299,299,L,116.21,346f7cff80ca3ab37dc5bb4fe5d9914b
Normal-6115.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6115.png,Normal/images/Normal-6115.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6115.png,Normal/masks/Normal-6115.png,True,False,299,299,L,120.96,0b550b8478e57ee9b3a83e2e697b053b
Normal-6116.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6116.png,Normal/images/Normal-6116.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6116.png,Normal/masks/Normal-6116.png,True,False,299,299,L,122.19,7fe6ab2f147d96324cc5951549d734e0
Normal-6117.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6117.png,Normal/images/Normal-6117.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6117.png,Normal/masks/Normal-6117.png,True,False,299,299,L,164.17,dedf88c0a6b46ea7ce8316ca06511a56
Normal-6118.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6118.png,Normal/images/Normal-6118.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6118.png,Normal/masks/Normal-6118.png,True,False,299,299,L,174.32,2cfb88a9ceb5cb24237f55daf23331f1
Normal-6119.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6119.png,Normal/images/Normal-6119.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6119.png,Normal/masks/Normal-6119.png,True,False,299,299,L,141.72,f76667d4e77ed0e6972fa3b3a9c99cb5
Normal-612.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-612.png,Normal/images/Normal-612.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-612.png,Normal/masks/Normal-612.png,True,False,299,299,L,123.48,414ef0185b01b617ec6d086cfa4c2345
Normal-6120.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6120.png,Normal/images/Normal-6120.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6120.png,Normal/masks/Normal-6120.png,True,False,299,299,L,112.95,f36666cb52e474ba69578d9b2fa380b5
Normal-6121.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6121.png,Normal/images/Normal-6121.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6121.png,Normal/masks/Normal-6121.png,True,False,299,299,L,154.25,bd043951e068ea09ce0d519ece0f6e2f
Normal-6122.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6122.png,Normal/images/Normal-6122.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6122.png,Normal/masks/Normal-6122.png,True,False,299,299,L,140.79,07aa00b8b5bcd4a5a2414837a0b77f79
Normal-6123.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6123.png,Normal/images/Normal-6123.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6123.png,Normal/masks/Normal-6123.png,True,False,299,299,L,128.43,ca4f5b4c5a917527eccc8f60368b729e
Normal-6124.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6124.png,Normal/images/Normal-6124.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6124.png,Normal/masks/Normal-6124.png,True,False,299,299,L,139.25,ef74e95e4fca905717c7023f2e91f99b
Normal-6125.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6125.png,Normal/images/Normal-6125.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6125.png,Normal/masks/Normal-6125.png,True,False,299,299,L,113.83,70b6d3e1b7cfb5d91698c4c2a371818f
Normal-6126.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6126.png,Normal/images/Normal-6126.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6126.png,Normal/masks/Normal-6126.png,True,False,299,299,L,131.3,d96e0f7575e09bf385a7453f2c61220d
Normal-6127.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6127.png,Normal/images/Normal-6127.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6127.png,Normal/masks/Normal-6127.png,True,False,299,299,L,115.96,d68f9e11da1fb6e6d6cf92795748139c
Normal-6128.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6128.png,Normal/images/Normal-6128.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6128.png,Normal/masks/Normal-6128.png,True,False,299,299,L,141.71,f6582a396e223925cda27d0841b318fd
Normal-6129.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6129.png,Normal/images/Normal-6129.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6129.png,Normal/masks/Normal-6129.png,True,False,299,299,L,152.08,485f0800f6db86fd519eeaf69238c993
Normal-613.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-613.png,Normal/images/Normal-613.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-613.png,Normal/masks/Normal-613.png,True,False,299,299,L,116.18,5f72a5b6783914d9b63c1aa9ea13340c
Normal-6130.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6130.png,Normal/images/Normal-6130.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6130.png,Normal/masks/Normal-6130.png,True,False,299,299,L,101.4,d3e507cec230b8dd2a179c6209871def
Normal-6131.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6131.png,Normal/images/Normal-6131.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6131.png,Normal/masks/Normal-6131.png,True,False,299,299,L,126.08,469183f6727d541ca5f2f48e954a6c3b
Normal-6132.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6132.png,Normal/images/Normal-6132.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6132.png,Normal/masks/Normal-6132.png,True,False,299,299,L,118.1,e38ab6f20e7a0a39f66ea0bc627570a8
Normal-6133.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6133.png,Normal/images/Normal-6133.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6133.png,Normal/masks/Normal-6133.png,True,False,299,299,L,167.15,ad5f60c80c6a1be5b4d6b3ab9c69994d
Normal-6134.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6134.png,Normal/images/Normal-6134.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6134.png,Normal/masks/Normal-6134.png,True,False,299,299,L,117.22,6347233144ea55f3a5b90de702a89320
Normal-6135.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6135.png,Normal/images/Normal-6135.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6135.png,Normal/masks/Normal-6135.png,True,False,299,299,L,117.12,10dc1f39c70a5d0d0e3813d7801da339
Normal-6136.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6136.png,Normal/images/Normal-6136.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6136.png,Normal/masks/Normal-6136.png,True,False,299,299,L,105.18,f647d9b671271cb9a46f343132ccc2f1
Normal-6137.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6137.png,Normal/images/Normal-6137.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6137.png,Normal/masks/Normal-6137.png,True,False,299,299,L,100.79,8c29355cfabee35fbce7d76574da69ba
Normal-6138.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6138.png,Normal/images/Normal-6138.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6138.png,Normal/masks/Normal-6138.png,True,False,299,299,L,117.94,431d4494831c504b2f5f91be84020d55
Normal-6139.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6139.png,Normal/images/Normal-6139.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6139.png,Normal/masks/Normal-6139.png,True,False,299,299,L,124.61,080f412c83b417ef7ce62783d7c9d4b2
Normal-614.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-614.png,Normal/images/Normal-614.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-614.png,Normal/masks/Normal-614.png,True,False,299,299,L,128.44,21e582ef95b4913402afc760f4283334
Normal-6140.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6140.png,Normal/images/Normal-6140.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6140.png,Normal/masks/Normal-6140.png,True,False,299,299,L,90.24,1d529bd3ba2ae46b3aaa9f8569230640
Normal-6141.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6141.png,Normal/images/Normal-6141.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6141.png,Normal/masks/Normal-6141.png,True,False,299,299,L,153.82,091e58208b67e6a93c5f431687edf16c
Normal-6142.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6142.png,Normal/images/Normal-6142.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6142.png,Normal/masks/Normal-6142.png,True,False,299,299,L,174.86,b60c928502ef8c26fb36424fdbdad762
Normal-6143.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6143.png,Normal/images/Normal-6143.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6143.png,Normal/masks/Normal-6143.png,True,False,299,299,L,127.51,9946562204d4fdbf58f470e8f0e0ead5
Normal-6144.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6144.png,Normal/images/Normal-6144.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6144.png,Normal/masks/Normal-6144.png,True,False,299,299,L,112.99,8108997fd12f4ec73284e54645e56b9a
Normal-6145.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6145.png,Normal/images/Normal-6145.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6145.png,Normal/masks/Normal-6145.png,True,False,299,299,L,158.66,bc7d5016b364737c68c8bb38c6250d8b
Normal-6146.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6146.png,Normal/images/Normal-6146.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6146.png,Normal/masks/Normal-6146.png,True,False,299,299,L,115.15,3724718112ed6e0338fbbb7391b60412
Normal-6147.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6147.png,Normal/images/Normal-6147.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6147.png,Normal/masks/Normal-6147.png,True,False,299,299,L,131.86,dcd4f9d03adba2222c0a2bca0f07ab98
Normal-6148.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6148.png,Normal/images/Normal-6148.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6148.png,Normal/masks/Normal-6148.png,True,False,299,299,L,105.41,ae79bfbd5b15912a243148c2d3cd78f7
Normal-6149.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6149.png,Normal/images/Normal-6149.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6149.png,Normal/masks/Normal-6149.png,True,False,299,299,L,104.82,e8173c02cca359ff1fd819a15cd7b325
Normal-615.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-615.png,Normal/images/Normal-615.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-615.png,Normal/masks/Normal-615.png,True,False,299,299,L,115.29,4f8b2157f9244b82eb1d28bf166f087d
Normal-6150.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6150.png,Normal/images/Normal-6150.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6150.png,Normal/masks/Normal-6150.png,True,False,299,299,L,112.57,82cb6956742ab2d16afa415ba7a8e3d5
Normal-6151.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6151.png,Normal/images/Normal-6151.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6151.png,Normal/masks/Normal-6151.png,True,False,299,299,L,178.36,394a5f19aa8f48fd5d879dbd3b164447
Normal-6152.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6152.png,Normal/images/Normal-6152.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6152.png,Normal/masks/Normal-6152.png,True,False,299,299,L,113.45,01ab298f581329330f835b8cb6d5c9f9
Normal-6153.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6153.png,Normal/images/Normal-6153.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6153.png,Normal/masks/Normal-6153.png,True,False,299,299,L,114.19,2a68af1661e87ea0a6677cf86f3025d2
Normal-6154.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6154.png,Normal/images/Normal-6154.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6154.png,Normal/masks/Normal-6154.png,True,False,299,299,L,145.53,2fcea865e83e5841ec81a4903b18807b
Normal-6155.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6155.png,Normal/images/Normal-6155.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6155.png,Normal/masks/Normal-6155.png,True,False,299,299,L,101.18,0da13e11bc09e31b95ca6ecfaa537a17
Normal-6156.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6156.png,Normal/images/Normal-6156.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6156.png,Normal/masks/Normal-6156.png,True,False,299,299,L,141.53,d4b29907a371e336949849836c09ccca
Normal-6157.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6157.png,Normal/images/Normal-6157.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6157.png,Normal/masks/Normal-6157.png,True,False,299,299,L,132.85,30ea3840e4157cfc3babef6cf1fed3d3
Normal-6158.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6158.png,Normal/images/Normal-6158.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6158.png,Normal/masks/Normal-6158.png,True,False,299,299,L,96.12,82d57e65c70952cb4d9dd7c7530db72e
Normal-6159.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6159.png,Normal/images/Normal-6159.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6159.png,Normal/masks/Normal-6159.png,True,False,299,299,L,98.9,09a29f69dcd071a01638458764c538e7
Normal-616.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-616.png,Normal/images/Normal-616.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-616.png,Normal/masks/Normal-616.png,True,False,299,299,L,130.37,193c32eed0a66af00fffe67c687408aa
Normal-6160.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6160.png,Normal/images/Normal-6160.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6160.png,Normal/masks/Normal-6160.png,True,False,299,299,L,164.42,c7c7503549c143f88bb0079e6fb8addd
Normal-6161.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6161.png,Normal/images/Normal-6161.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6161.png,Normal/masks/Normal-6161.png,True,False,299,299,L,120.27,aa6a5f05764119a9db5fdb6e028f6531
Normal-6162.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6162.png,Normal/images/Normal-6162.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6162.png,Normal/masks/Normal-6162.png,True,False,299,299,L,121.34,088d8f698d0b7fd3d0666a243492b6d8
Normal-6163.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6163.png,Normal/images/Normal-6163.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6163.png,Normal/masks/Normal-6163.png,True,False,299,299,L,122.11,59759e6d98b95a4236bf052bdf0ad79a
Normal-6164.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6164.png,Normal/images/Normal-6164.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6164.png,Normal/masks/Normal-6164.png,True,False,299,299,L,132.97,e86f579a996bfa5da951cdf9077c75f6
Normal-6165.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6165.png,Normal/images/Normal-6165.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6165.png,Normal/masks/Normal-6165.png,True,False,299,299,L,114.77,729d16e93f624d5b1462a6c0f5b7f30a
Normal-6166.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6166.png,Normal/images/Normal-6166.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6166.png,Normal/masks/Normal-6166.png,True,False,299,299,L,155.43,cf6a2e01c5a01d2ac8b47436c61d57f2
Normal-6167.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6167.png,Normal/images/Normal-6167.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6167.png,Normal/masks/Normal-6167.png,True,False,299,299,L,115.95,11c09b864679f9ab02620afa249ac2de
Normal-6168.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6168.png,Normal/images/Normal-6168.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6168.png,Normal/masks/Normal-6168.png,True,False,299,299,L,173.82,1ec4a54c03650e4c771d93895d70cd66
Normal-6169.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6169.png,Normal/images/Normal-6169.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6169.png,Normal/masks/Normal-6169.png,True,False,299,299,L,169.08,01abba6603c7fb3d9cb18e7e46eb4ab6
Normal-617.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-617.png,Normal/images/Normal-617.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-617.png,Normal/masks/Normal-617.png,True,False,299,299,L,132.41,e7efb01b306e8f00dfcf6ca09f0e17e9
Normal-6170.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6170.png,Normal/images/Normal-6170.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6170.png,Normal/masks/Normal-6170.png,True,False,299,299,L,155.16,580f8dd28544e55adc94204ca3a698fb
Normal-6171.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6171.png,Normal/images/Normal-6171.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6171.png,Normal/masks/Normal-6171.png,True,False,299,299,L,128.76,902824ee5b8a12c809ed32ae84dbbea2
Normal-6172.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6172.png,Normal/images/Normal-6172.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6172.png,Normal/masks/Normal-6172.png,True,False,299,299,L,151.19,8f26e7807be8c100acda329007db8bca
Normal-6173.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6173.png,Normal/images/Normal-6173.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6173.png,Normal/masks/Normal-6173.png,True,False,299,299,L,111.7,e8bbce5a43ffd9509a3e2e39b443e75c
Normal-6174.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6174.png,Normal/images/Normal-6174.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6174.png,Normal/masks/Normal-6174.png,True,False,299,299,L,169.24,4d7b40308c5fe2e6bfa38722587c619d
Normal-6175.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6175.png,Normal/images/Normal-6175.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6175.png,Normal/masks/Normal-6175.png,True,False,299,299,L,119.34,f369b3c6c84673997277c6b8b3380d76
Normal-6176.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6176.png,Normal/images/Normal-6176.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6176.png,Normal/masks/Normal-6176.png,True,False,299,299,L,130.82,9e637f8be61872b229dd710f5e216571
Normal-6177.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6177.png,Normal/images/Normal-6177.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6177.png,Normal/masks/Normal-6177.png,True,False,299,299,L,171.39,08feb5ea64c1593aeec77f8f45f22c55
Normal-6178.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6178.png,Normal/images/Normal-6178.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6178.png,Normal/masks/Normal-6178.png,True,False,299,299,L,103.84,2b2064387c946f7bc0446c15d28ae06f
Normal-6179.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6179.png,Normal/images/Normal-6179.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6179.png,Normal/masks/Normal-6179.png,True,False,299,299,L,125.35,f63e8169d55f30a8bd0044f3e05923dc
Normal-618.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-618.png,Normal/images/Normal-618.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-618.png,Normal/masks/Normal-618.png,True,False,299,299,L,126.47,2840de22883e718c01754261921083cf
Normal-6180.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6180.png,Normal/images/Normal-6180.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6180.png,Normal/masks/Normal-6180.png,True,False,299,299,L,102.46,fdafa4bd7de05aeda6b5e051be8695e0
Normal-6181.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6181.png,Normal/images/Normal-6181.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6181.png,Normal/masks/Normal-6181.png,True,False,299,299,L,134.58,2d4f60314206af052117b16bf9e9c1b0
Normal-6182.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6182.png,Normal/images/Normal-6182.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6182.png,Normal/masks/Normal-6182.png,True,False,299,299,L,103.84,9bf0479ea4973ad0ada1e4d329fb0e5f
Normal-6183.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6183.png,Normal/images/Normal-6183.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6183.png,Normal/masks/Normal-6183.png,True,False,299,299,L,104.39,1bcc62a604b1b438881458876da753fb
Normal-6184.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6184.png,Normal/images/Normal-6184.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6184.png,Normal/masks/Normal-6184.png,True,False,299,299,L,165.27,a884d0692ac0a7902f3583873fd8f14d
Normal-6185.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6185.png,Normal/images/Normal-6185.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6185.png,Normal/masks/Normal-6185.png,True,False,299,299,L,122.35,a7f2a1b663ad2986b2083d5a28fdfc6b
Normal-6186.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6186.png,Normal/images/Normal-6186.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6186.png,Normal/masks/Normal-6186.png,True,False,299,299,L,119.91,3de54a904e65abb2725caddf67b2b232
Normal-6187.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6187.png,Normal/images/Normal-6187.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6187.png,Normal/masks/Normal-6187.png,True,False,299,299,L,102.92,913b9091e252e4e6d73d61d7aa9cb3ae
Normal-6188.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6188.png,Normal/images/Normal-6188.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6188.png,Normal/masks/Normal-6188.png,True,False,299,299,L,167.68,d6ac4cef73ebc74c0a948b78d68a67f8
Normal-6189.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6189.png,Normal/images/Normal-6189.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6189.png,Normal/masks/Normal-6189.png,True,False,299,299,L,118.15,5b4e9c537faaf40c776033b496021fd3
Normal-619.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-619.png,Normal/images/Normal-619.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-619.png,Normal/masks/Normal-619.png,True,False,299,299,L,112.76,59476dead18b54e6e9a9e1c6edbebdba
Normal-6190.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6190.png,Normal/images/Normal-6190.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6190.png,Normal/masks/Normal-6190.png,True,False,299,299,L,157.73,4bd63056a49428e26b83f2ea0071636c
Normal-6191.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6191.png,Normal/images/Normal-6191.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6191.png,Normal/masks/Normal-6191.png,True,False,299,299,L,110.41,3ae199317a71a91d4e7c9916da0aa10d
Normal-6192.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6192.png,Normal/images/Normal-6192.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6192.png,Normal/masks/Normal-6192.png,True,False,299,299,L,144.98,3a0b190d64fc508d75c23b69449af899
Normal-6193.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6193.png,Normal/images/Normal-6193.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6193.png,Normal/masks/Normal-6193.png,True,False,299,299,L,92.13,38a89272f02073f9589b7999626cba05
Normal-6194.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6194.png,Normal/images/Normal-6194.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6194.png,Normal/masks/Normal-6194.png,True,False,299,299,L,82.12,c8f620452f2f51dfed14db1fd4e55cfd
Normal-6195.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6195.png,Normal/images/Normal-6195.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6195.png,Normal/masks/Normal-6195.png,True,False,299,299,L,113.53,5c827aa5923efee2166468bca8fe85e0
Normal-6196.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6196.png,Normal/images/Normal-6196.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6196.png,Normal/masks/Normal-6196.png,True,False,299,299,L,149.65,a6690494f98a48d04951cb072cb27e86
Normal-6197.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6197.png,Normal/images/Normal-6197.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6197.png,Normal/masks/Normal-6197.png,True,False,299,299,L,85.82,e1dac05ec5226852173020b667a85868
Normal-6198.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6198.png,Normal/images/Normal-6198.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6198.png,Normal/masks/Normal-6198.png,True,False,299,299,L,164.96,8eeeee65a53fa82580a3203b84faeb7c
Normal-6199.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6199.png,Normal/images/Normal-6199.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6199.png,Normal/masks/Normal-6199.png,True,False,299,299,L,109.52,1afa6948bbef012d84ed1fb7e39d9046
Normal-62.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-62.png,Normal/images/Normal-62.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-62.png,Normal/masks/Normal-62.png,True,False,299,299,L,123.79,4b663defe17b94aa3cbd085f65863ac2
Normal-620.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-620.png,Normal/images/Normal-620.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-620.png,Normal/masks/Normal-620.png,True,False,299,299,L,110.25,017aa476c1580af812122dc96fdbd3d6
Normal-6200.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6200.png,Normal/images/Normal-6200.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6200.png,Normal/masks/Normal-6200.png,True,False,299,299,L,95.61,fdf1cf5da11d53a789f1fd20740aaea2
Normal-6201.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6201.png,Normal/images/Normal-6201.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6201.png,Normal/masks/Normal-6201.png,True,False,299,299,L,129.69,1254227f9bf65cff6a9bbcba50055721
Normal-6202.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6202.png,Normal/images/Normal-6202.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6202.png,Normal/masks/Normal-6202.png,True,False,299,299,L,155.55,819a848eb809763700a7b2ab00e338ec
Normal-6203.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6203.png,Normal/images/Normal-6203.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6203.png,Normal/masks/Normal-6203.png,True,False,299,299,L,166.63,4d5cc8820443a46eea53818f447e4a62
Normal-6204.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6204.png,Normal/images/Normal-6204.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6204.png,Normal/masks/Normal-6204.png,True,False,299,299,L,121.84,cb4744b66a6a5e6c567687b59bb6e9d4
Normal-6205.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6205.png,Normal/images/Normal-6205.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6205.png,Normal/masks/Normal-6205.png,True,False,299,299,L,143.75,5c1c682358e72e6b155c7a3bead22653
Normal-6206.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6206.png,Normal/images/Normal-6206.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6206.png,Normal/masks/Normal-6206.png,True,False,299,299,L,156.18,fd2277d3e1b13f5014c28e364897c59a
Normal-6207.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6207.png,Normal/images/Normal-6207.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6207.png,Normal/masks/Normal-6207.png,True,False,299,299,L,125.39,84c6c393b5bf4b320fdf826a1b37a62c
Normal-6208.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6208.png,Normal/images/Normal-6208.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6208.png,Normal/masks/Normal-6208.png,True,False,299,299,L,125.22,2cd3b6f7b7638dbecce810df87da5f28
Normal-6209.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6209.png,Normal/images/Normal-6209.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6209.png,Normal/masks/Normal-6209.png,True,False,299,299,L,125.48,7c9f42c175d9c0d2c984e6959de5f9de
Normal-621.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-621.png,Normal/images/Normal-621.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-621.png,Normal/masks/Normal-621.png,True,False,299,299,L,119.31,0ceb46458a93e66c6eb356716408ba60
Normal-6210.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6210.png,Normal/images/Normal-6210.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6210.png,Normal/masks/Normal-6210.png,True,False,299,299,L,104.94,0d83d9da889e7ee23ff3a0e1c160d0ea
Normal-6211.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6211.png,Normal/images/Normal-6211.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6211.png,Normal/masks/Normal-6211.png,True,False,299,299,L,124.14,7483ed78452c7fd52977f3b378bb5353
Normal-6212.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6212.png,Normal/images/Normal-6212.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6212.png,Normal/masks/Normal-6212.png,True,False,299,299,L,93.01,4f372b1f2095c263524fb229483067d7
Normal-6213.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6213.png,Normal/images/Normal-6213.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6213.png,Normal/masks/Normal-6213.png,True,False,299,299,L,176.83,0f9cdef8e95d564e6687f1eb109c8a1a
Normal-6214.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6214.png,Normal/images/Normal-6214.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6214.png,Normal/masks/Normal-6214.png,True,False,299,299,L,166.73,3bda3d1b76b319005c04c4b50a6ab012
Normal-6215.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6215.png,Normal/images/Normal-6215.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6215.png,Normal/masks/Normal-6215.png,True,False,299,299,L,124.87,a9460fab82eed22456879a483164338c
Normal-6216.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6216.png,Normal/images/Normal-6216.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6216.png,Normal/masks/Normal-6216.png,True,False,299,299,L,86.42,6d03dac3e716300da1a21d0a3aaf36c7
Normal-6217.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6217.png,Normal/images/Normal-6217.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6217.png,Normal/masks/Normal-6217.png,True,False,299,299,L,103.31,1c6092175fbeaf33541bc5b4f1a8556f
Normal-6218.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6218.png,Normal/images/Normal-6218.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6218.png,Normal/masks/Normal-6218.png,True,False,299,299,L,154.74,4b337a1dbc8a4afad684cb4b3d48eb0b
Normal-6219.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6219.png,Normal/images/Normal-6219.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6219.png,Normal/masks/Normal-6219.png,True,False,299,299,L,67.5,91ee93768194913504d1c3f10880b498
Normal-622.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-622.png,Normal/images/Normal-622.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-622.png,Normal/masks/Normal-622.png,True,False,299,299,L,107.05,dcb32581988f05d5d710abeb1d42fdb9
Normal-6220.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6220.png,Normal/images/Normal-6220.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6220.png,Normal/masks/Normal-6220.png,True,False,299,299,L,132.16,fc5cc3d385134116d866c00b56301b32
Normal-6221.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6221.png,Normal/images/Normal-6221.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6221.png,Normal/masks/Normal-6221.png,True,False,299,299,L,150.65,bef1e8d37a21d0bb412b32f26062a851
Normal-6222.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6222.png,Normal/images/Normal-6222.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6222.png,Normal/masks/Normal-6222.png,True,False,299,299,L,110.35,f4af3fa7644c77f04b07fcdac8e7c884
Normal-6223.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6223.png,Normal/images/Normal-6223.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6223.png,Normal/masks/Normal-6223.png,True,False,299,299,L,110.12,736d275c1d1730cb410b0361cc7c5bec
Normal-6224.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6224.png,Normal/images/Normal-6224.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6224.png,Normal/masks/Normal-6224.png,True,False,299,299,L,116.66,ef586cc4469114c10f5d0a7e203e5069
Normal-6225.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6225.png,Normal/images/Normal-6225.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6225.png,Normal/masks/Normal-6225.png,True,False,299,299,L,149.27,512cbc73c6426d2672a3fb79aba8b10b
Normal-6226.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6226.png,Normal/images/Normal-6226.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6226.png,Normal/masks/Normal-6226.png,True,False,299,299,L,150.25,8ef7cb335a496185fd554f7ea43858e2
Normal-6227.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6227.png,Normal/images/Normal-6227.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6227.png,Normal/masks/Normal-6227.png,True,False,299,299,L,114.14,d5fcda7d18dcd7a26128aa989a963108
Normal-6228.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6228.png,Normal/images/Normal-6228.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6228.png,Normal/masks/Normal-6228.png,True,False,299,299,L,133.46,38abf2425f333019fddaa0576bd0568f
Normal-6229.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6229.png,Normal/images/Normal-6229.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6229.png,Normal/masks/Normal-6229.png,True,False,299,299,L,148.45,f2e72176489126b7d659571bff0db6a4
Normal-623.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-623.png,Normal/images/Normal-623.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-623.png,Normal/masks/Normal-623.png,True,False,299,299,L,97.58,e62c932db3f8a56dcad092f20e8167d7
Normal-6230.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6230.png,Normal/images/Normal-6230.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6230.png,Normal/masks/Normal-6230.png,True,False,299,299,L,120.57,1f7cd8971fede44061c75d51b193441e
Normal-6231.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6231.png,Normal/images/Normal-6231.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6231.png,Normal/masks/Normal-6231.png,True,False,299,299,L,117.72,9c55a3612a5348632746bb88c5f46261
Normal-6232.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6232.png,Normal/images/Normal-6232.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6232.png,Normal/masks/Normal-6232.png,True,False,299,299,L,135.46,5df4102312ad75b618bacb397f35e820
Normal-6233.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6233.png,Normal/images/Normal-6233.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6233.png,Normal/masks/Normal-6233.png,True,False,299,299,L,113.56,65c34da01b837e34b0fe75c34a7affdd
Normal-6234.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6234.png,Normal/images/Normal-6234.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6234.png,Normal/masks/Normal-6234.png,True,False,299,299,L,172.83,b34aa000835bbddf874804e1cb8a3b35
Normal-6235.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6235.png,Normal/images/Normal-6235.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6235.png,Normal/masks/Normal-6235.png,True,False,299,299,L,162.23,4c731893da54a0217cc2d5f1487d2ad5
Normal-6236.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6236.png,Normal/images/Normal-6236.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6236.png,Normal/masks/Normal-6236.png,True,False,299,299,L,104.97,a663dd315630712c71ffe23e09c21123
Normal-6237.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6237.png,Normal/images/Normal-6237.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6237.png,Normal/masks/Normal-6237.png,True,False,299,299,L,110.38,439ad4eb1bd9de40a1ed27ab7a43ef7a
Normal-6238.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6238.png,Normal/images/Normal-6238.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6238.png,Normal/masks/Normal-6238.png,True,False,299,299,L,136.55,3ab6c3b58e2a2fc81aaef4de9fc8d8f2
Normal-6239.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6239.png,Normal/images/Normal-6239.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6239.png,Normal/masks/Normal-6239.png,True,False,299,299,L,87.88,4bf6b6892fbe40359bda4f5201321229
Normal-624.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-624.png,Normal/images/Normal-624.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-624.png,Normal/masks/Normal-624.png,True,False,299,299,L,105.11,e46dd56ba938eac607668df0ef9b80cb
Normal-6240.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6240.png,Normal/images/Normal-6240.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6240.png,Normal/masks/Normal-6240.png,True,False,299,299,L,126.22,00ac2a1b326cd5421c4b6c962cf50725
Normal-6241.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6241.png,Normal/images/Normal-6241.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6241.png,Normal/masks/Normal-6241.png,True,False,299,299,L,117.05,737498bfde898f086728109cf553561b
Normal-6242.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6242.png,Normal/images/Normal-6242.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6242.png,Normal/masks/Normal-6242.png,True,False,299,299,L,112.28,af96f37d90c1493977652beb3794b6fc
Normal-6243.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6243.png,Normal/images/Normal-6243.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6243.png,Normal/masks/Normal-6243.png,True,False,299,299,L,159.32,c6c863281e89b1a787945c1214c59563
Normal-6244.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6244.png,Normal/images/Normal-6244.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6244.png,Normal/masks/Normal-6244.png,True,False,299,299,L,117.47,74813971b76f66cea9f7f7468da31daf
Normal-6245.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6245.png,Normal/images/Normal-6245.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6245.png,Normal/masks/Normal-6245.png,True,False,299,299,L,138.91,1125cdb0aaa5ef9b26a00b096b9ac877
Normal-6246.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6246.png,Normal/images/Normal-6246.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6246.png,Normal/masks/Normal-6246.png,True,False,299,299,L,118.71,92a10a938e04a969582a772638b97340
Normal-6247.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6247.png,Normal/images/Normal-6247.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6247.png,Normal/masks/Normal-6247.png,True,False,299,299,L,161.5,d1a685237bedebe81a97868869019699
Normal-6248.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6248.png,Normal/images/Normal-6248.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6248.png,Normal/masks/Normal-6248.png,True,False,299,299,L,132.78,9e6e88d176aedc58789b733df53b2e41
Normal-6249.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6249.png,Normal/images/Normal-6249.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6249.png,Normal/masks/Normal-6249.png,True,False,299,299,L,108.88,e7cada4dc8f356f3bf27528de69c658d
Normal-625.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-625.png,Normal/images/Normal-625.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-625.png,Normal/masks/Normal-625.png,True,False,299,299,L,109.01,f5efacff0ccb704a21c48788cbaaedb2
Normal-6250.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6250.png,Normal/images/Normal-6250.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6250.png,Normal/masks/Normal-6250.png,True,False,299,299,L,157.16,99d38ae4008e755ab01a384163b9bb69
Normal-6251.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6251.png,Normal/images/Normal-6251.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6251.png,Normal/masks/Normal-6251.png,True,False,299,299,L,164.96,c2177c39e6e166d96a3720ebe60f5e8e
Normal-6252.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6252.png,Normal/images/Normal-6252.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6252.png,Normal/masks/Normal-6252.png,True,False,299,299,L,125.25,7b4935d4b97365fb74c62550d3802c5b
Normal-6253.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6253.png,Normal/images/Normal-6253.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6253.png,Normal/masks/Normal-6253.png,True,False,299,299,L,106.44,24a271e1bb76a71f4dac8db783137cd9
Normal-6254.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6254.png,Normal/images/Normal-6254.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6254.png,Normal/masks/Normal-6254.png,True,False,299,299,L,108.56,9c2b6a8eb1713abfc0f75a8eae8b4d66
Normal-6255.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6255.png,Normal/images/Normal-6255.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6255.png,Normal/masks/Normal-6255.png,True,False,299,299,L,99.37,169d2c258993b6cd8ea741287865ab63
Normal-6256.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6256.png,Normal/images/Normal-6256.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6256.png,Normal/masks/Normal-6256.png,True,False,299,299,L,118.13,90b297b65f382ae1821da7743f6e9b61
Normal-6257.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6257.png,Normal/images/Normal-6257.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6257.png,Normal/masks/Normal-6257.png,True,False,299,299,L,99.13,906a38e1f46255d1ee4a2e44d8aa4128
Normal-6258.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6258.png,Normal/images/Normal-6258.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6258.png,Normal/masks/Normal-6258.png,True,False,299,299,L,161.95,e18d5c98619dc8dc9e087a4f1f258811
Normal-6259.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6259.png,Normal/images/Normal-6259.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6259.png,Normal/masks/Normal-6259.png,True,False,299,299,L,150.71,493f1f51bb9a37a2fb3637e4da348263
Normal-626.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-626.png,Normal/images/Normal-626.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-626.png,Normal/masks/Normal-626.png,True,False,299,299,L,113.03,864e08f3ea39a99e2d38fe84021cadad
Normal-6260.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6260.png,Normal/images/Normal-6260.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6260.png,Normal/masks/Normal-6260.png,True,False,299,299,L,170.41,dbeb665a77d64467f6d8116462870b82
Normal-6261.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6261.png,Normal/images/Normal-6261.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6261.png,Normal/masks/Normal-6261.png,True,False,299,299,L,183.02,25004185ab0d7fd3abdb4f4a855bdd2c
Normal-6262.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6262.png,Normal/images/Normal-6262.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6262.png,Normal/masks/Normal-6262.png,True,False,299,299,L,131.3,ae6397c4555b267b2447bcd615d4be2e
Normal-6263.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6263.png,Normal/images/Normal-6263.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6263.png,Normal/masks/Normal-6263.png,True,False,299,299,L,111.24,0caf76c0de6f943f4793bd22824919bc
Normal-6264.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6264.png,Normal/images/Normal-6264.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6264.png,Normal/masks/Normal-6264.png,True,False,299,299,L,116.99,c0f7f9df790b50c617547f7a7418f592
Normal-6265.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6265.png,Normal/images/Normal-6265.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6265.png,Normal/masks/Normal-6265.png,True,False,299,299,L,165.58,0e6e7c1a966c4bd5a520bea2b3355ed7
Normal-6266.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6266.png,Normal/images/Normal-6266.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6266.png,Normal/masks/Normal-6266.png,True,False,299,299,L,100.62,e039aa725e836a1cd8ec4785b60d5d0d
Normal-6267.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6267.png,Normal/images/Normal-6267.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6267.png,Normal/masks/Normal-6267.png,True,False,299,299,L,93.94,e13d4b5280b1996d300dddc93d5e152d
Normal-6268.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6268.png,Normal/images/Normal-6268.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6268.png,Normal/masks/Normal-6268.png,True,False,299,299,L,100.14,464e4d82c14b11dc07f2f7beba27d5d4
Normal-6269.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6269.png,Normal/images/Normal-6269.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6269.png,Normal/masks/Normal-6269.png,True,False,299,299,L,102.36,f9d207c035ff6d5cb521df6d647a6de7
Normal-627.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-627.png,Normal/images/Normal-627.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-627.png,Normal/masks/Normal-627.png,True,False,299,299,L,104.36,198ab995b4c7d52c0b1671df50290a95
Normal-6270.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6270.png,Normal/images/Normal-6270.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6270.png,Normal/masks/Normal-6270.png,True,False,299,299,L,139.43,657bd29b212036cd61cfa9cf86139961
Normal-6271.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6271.png,Normal/images/Normal-6271.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6271.png,Normal/masks/Normal-6271.png,True,False,299,299,L,109.34,dfb75bcb435060ea50114d00c98ab2fb
Normal-6272.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6272.png,Normal/images/Normal-6272.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6272.png,Normal/masks/Normal-6272.png,True,False,299,299,L,109.81,0d40fc9a51dd2b8f488c8fef424f6c82
Normal-6273.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6273.png,Normal/images/Normal-6273.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6273.png,Normal/masks/Normal-6273.png,True,False,299,299,L,173.16,900240d9d9f99ed5861cfc191864a570
Normal-6274.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6274.png,Normal/images/Normal-6274.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6274.png,Normal/masks/Normal-6274.png,True,False,299,299,L,162.56,bc73b941c5aa88ef59d7b998a7ff256d
Normal-6275.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6275.png,Normal/images/Normal-6275.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6275.png,Normal/masks/Normal-6275.png,True,False,299,299,L,121.75,eec270bbdb6c9706ed7912240429cb90
Normal-6276.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6276.png,Normal/images/Normal-6276.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6276.png,Normal/masks/Normal-6276.png,True,False,299,299,L,188.46,f1fd9155503a45f79a322a49ae333247
Normal-6277.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6277.png,Normal/images/Normal-6277.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6277.png,Normal/masks/Normal-6277.png,True,False,299,299,L,112.13,1ea1ee1026220a28a8d6da83a5066686
Normal-6278.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6278.png,Normal/images/Normal-6278.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6278.png,Normal/masks/Normal-6278.png,True,False,299,299,L,118.04,a2711f02ee1eddb94a88c1a6282a7d6a
Normal-6279.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6279.png,Normal/images/Normal-6279.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6279.png,Normal/masks/Normal-6279.png,True,False,299,299,L,121.98,32896038978154372149012838d6f603
Normal-628.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-628.png,Normal/images/Normal-628.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-628.png,Normal/masks/Normal-628.png,True,False,299,299,L,130.61,2be14528ee7299a253cced847dab3d08
Normal-6280.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6280.png,Normal/images/Normal-6280.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6280.png,Normal/masks/Normal-6280.png,True,False,299,299,L,147.86,d4b62f6b22559f96e92530101072bcbc
Normal-6281.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6281.png,Normal/images/Normal-6281.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6281.png,Normal/masks/Normal-6281.png,True,False,299,299,L,108.19,ff2936574c5a751c83d1d18a774a1d32
Normal-6282.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6282.png,Normal/images/Normal-6282.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6282.png,Normal/masks/Normal-6282.png,True,False,299,299,L,171.96,e4078f11880ed2f751dfe07671e8e815
Normal-6283.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6283.png,Normal/images/Normal-6283.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6283.png,Normal/masks/Normal-6283.png,True,False,299,299,L,124.76,6d87c37d71c754fbbea7428a8c846cb6
Normal-6284.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6284.png,Normal/images/Normal-6284.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6284.png,Normal/masks/Normal-6284.png,True,False,299,299,L,86.53,9cec3b8ceea4b888962e4ee17eab06a3
Normal-6285.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6285.png,Normal/images/Normal-6285.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6285.png,Normal/masks/Normal-6285.png,True,False,299,299,L,105.4,9218532f4c9ce12a8928e3240777b4e7
Normal-6286.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6286.png,Normal/images/Normal-6286.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6286.png,Normal/masks/Normal-6286.png,True,False,299,299,L,176.78,a9551fd476f980a7f7990edaa49f1115
Normal-6287.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6287.png,Normal/images/Normal-6287.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6287.png,Normal/masks/Normal-6287.png,True,False,299,299,L,171.16,14436798d44bff38e4053873095aaf2d
Normal-6288.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6288.png,Normal/images/Normal-6288.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6288.png,Normal/masks/Normal-6288.png,True,False,299,299,L,115.71,caf9d5cbc96c7ece822d4110e57ca601
Normal-6289.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6289.png,Normal/images/Normal-6289.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6289.png,Normal/masks/Normal-6289.png,True,False,299,299,L,103.95,59091c8cebca107d06d81e6458570362
Normal-629.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-629.png,Normal/images/Normal-629.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-629.png,Normal/masks/Normal-629.png,True,False,299,299,L,121.6,babf1160907b57d0f3f2cf6b2e591365
Normal-6290.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6290.png,Normal/images/Normal-6290.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6290.png,Normal/masks/Normal-6290.png,True,False,299,299,L,129.47,95769c6f1aa84a36afc5281f13e9e02e
Normal-6291.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6291.png,Normal/images/Normal-6291.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6291.png,Normal/masks/Normal-6291.png,True,False,299,299,L,125.17,332f6af656ed42eaa2fba2a0607ad3ce
Normal-6292.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6292.png,Normal/images/Normal-6292.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6292.png,Normal/masks/Normal-6292.png,True,False,299,299,L,137.22,fc2504e6a4375681fc15f469370503a5
Normal-6293.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6293.png,Normal/images/Normal-6293.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6293.png,Normal/masks/Normal-6293.png,True,False,299,299,L,120.62,172d0dc8f4649dae1148369affa134f0
Normal-6294.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6294.png,Normal/images/Normal-6294.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6294.png,Normal/masks/Normal-6294.png,True,False,299,299,L,163.3,03f16712920c428fe796bab94c341098
Normal-6295.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6295.png,Normal/images/Normal-6295.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6295.png,Normal/masks/Normal-6295.png,True,False,299,299,L,169.77,7b6289a8722c4b82e2eee7d6cdc60dc6
Normal-6296.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6296.png,Normal/images/Normal-6296.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6296.png,Normal/masks/Normal-6296.png,True,False,299,299,L,157.35,462d606d4afc2000e17c4e45cc53f9ab
Normal-6297.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6297.png,Normal/images/Normal-6297.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6297.png,Normal/masks/Normal-6297.png,True,False,299,299,L,158.49,5a840a0bd1ef839a08bc480023060c99
Normal-6298.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6298.png,Normal/images/Normal-6298.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6298.png,Normal/masks/Normal-6298.png,True,False,299,299,L,144.5,f27f18ca2e0831fb9938554b7ad6adc1
Normal-6299.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6299.png,Normal/images/Normal-6299.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6299.png,Normal/masks/Normal-6299.png,True,False,299,299,L,134.5,166104abdaf963222d14d3f29a43d89e
Normal-63.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-63.png,Normal/images/Normal-63.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-63.png,Normal/masks/Normal-63.png,True,False,299,299,L,128.91,05b8f40ac0cb429a4d73e8ed764b62ce
Normal-630.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-630.png,Normal/images/Normal-630.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-630.png,Normal/masks/Normal-630.png,True,False,299,299,L,118.79,e99e89200180b17ce3da78a03cb6bb18
Normal-6300.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6300.png,Normal/images/Normal-6300.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6300.png,Normal/masks/Normal-6300.png,True,False,299,299,L,110.48,5fef274ae4f26248bd67bb897fd7edb1
Normal-6301.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6301.png,Normal/images/Normal-6301.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6301.png,Normal/masks/Normal-6301.png,True,False,299,299,L,116.98,5e70ecad9f29bbe14325945d6151db32
Normal-6302.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6302.png,Normal/images/Normal-6302.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6302.png,Normal/masks/Normal-6302.png,True,False,299,299,L,156.53,79f58f39bdc20230b9e340768e7437b9
Normal-6303.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6303.png,Normal/images/Normal-6303.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6303.png,Normal/masks/Normal-6303.png,True,False,299,299,L,115.83,8417b8ac4c9333e761aba78378b96fa0
Normal-6304.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6304.png,Normal/images/Normal-6304.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6304.png,Normal/masks/Normal-6304.png,True,False,299,299,L,98.52,212c67e15e9885f0609b5c0420dae355
Normal-6305.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6305.png,Normal/images/Normal-6305.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6305.png,Normal/masks/Normal-6305.png,True,False,299,299,L,109.8,276cbbd1dc51e8e5800e2b5ae9f55a87
Normal-6306.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6306.png,Normal/images/Normal-6306.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6306.png,Normal/masks/Normal-6306.png,True,False,299,299,L,126.63,d147982504942ddebc1920ac34804859
Normal-6307.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6307.png,Normal/images/Normal-6307.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6307.png,Normal/masks/Normal-6307.png,True,False,299,299,L,121.07,a4bf0ca6dbccfd00016f7cee8cf273c3
Normal-6308.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6308.png,Normal/images/Normal-6308.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6308.png,Normal/masks/Normal-6308.png,True,False,299,299,L,121.18,94a403182830ba10c42e3f45e6c3dd8e
Normal-6309.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6309.png,Normal/images/Normal-6309.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6309.png,Normal/masks/Normal-6309.png,True,False,299,299,L,160.55,1647b269facfbf158e888838c0702e06
Normal-631.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-631.png,Normal/images/Normal-631.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-631.png,Normal/masks/Normal-631.png,True,False,299,299,L,109.68,b4aff8a08abcd533f4b665440c0092f9
Normal-6310.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6310.png,Normal/images/Normal-6310.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6310.png,Normal/masks/Normal-6310.png,True,False,299,299,L,120.27,b760280c42dd19711a49ee87255fa2cc
Normal-6311.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6311.png,Normal/images/Normal-6311.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6311.png,Normal/masks/Normal-6311.png,True,False,299,299,L,164.11,3d4154e6e0deb712eb63930e8df9e66f
Normal-6312.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6312.png,Normal/images/Normal-6312.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6312.png,Normal/masks/Normal-6312.png,True,False,299,299,L,194.51,17b2796f4a9f1959ddbb0d74b5bf4698
Normal-6313.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6313.png,Normal/images/Normal-6313.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6313.png,Normal/masks/Normal-6313.png,True,False,299,299,L,93.61,0541356522139dbd9384d9713eed0401
Normal-6314.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6314.png,Normal/images/Normal-6314.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6314.png,Normal/masks/Normal-6314.png,True,False,299,299,L,135.04,43e2042d7a5cab4cc70055e62a0378a9
Normal-6315.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6315.png,Normal/images/Normal-6315.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6315.png,Normal/masks/Normal-6315.png,True,False,299,299,L,94.32,cf50679536458af8ead2f30217ed9d16
Normal-6316.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6316.png,Normal/images/Normal-6316.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6316.png,Normal/masks/Normal-6316.png,True,False,299,299,L,97.17,9ab6f47fe4aeda7ed33e085f569bc1c9
Normal-6317.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6317.png,Normal/images/Normal-6317.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6317.png,Normal/masks/Normal-6317.png,True,False,299,299,L,114.99,d756180a5b5a791dd9ddef8218a722f6
Normal-6318.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6318.png,Normal/images/Normal-6318.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6318.png,Normal/masks/Normal-6318.png,True,False,299,299,L,134.14,cd151b444a9001e7faa71b609fd5222f
Normal-6319.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6319.png,Normal/images/Normal-6319.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6319.png,Normal/masks/Normal-6319.png,True,False,299,299,L,132.46,033967b0cd2e01954467b848b48614ee
Normal-632.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-632.png,Normal/images/Normal-632.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-632.png,Normal/masks/Normal-632.png,True,False,299,299,L,121.95,ce93c33fc98ee7ba7c6b20f033d90747
Normal-6320.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6320.png,Normal/images/Normal-6320.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6320.png,Normal/masks/Normal-6320.png,True,False,299,299,L,110.36,e4ed9cdcbd82a84282a0c366c53d13e8
Normal-6321.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6321.png,Normal/images/Normal-6321.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6321.png,Normal/masks/Normal-6321.png,True,False,299,299,L,156.07,f5881e8403b1ec750d86ca2cbf2ab45a
Normal-6322.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6322.png,Normal/images/Normal-6322.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6322.png,Normal/masks/Normal-6322.png,True,False,299,299,L,128.18,a20c966b6cad9e1d7663478c46e497c6
Normal-6323.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6323.png,Normal/images/Normal-6323.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6323.png,Normal/masks/Normal-6323.png,True,False,299,299,L,123.38,a8f74cad5fa67d387c85753c0d8dde79
Normal-6324.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6324.png,Normal/images/Normal-6324.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6324.png,Normal/masks/Normal-6324.png,True,False,299,299,L,111.48,e562bff3b649fec413eceddfc9d3830c
Normal-6325.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6325.png,Normal/images/Normal-6325.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6325.png,Normal/masks/Normal-6325.png,True,False,299,299,L,120.23,aa4a866eafbe9dd2d98cb04a6a83b01d
Normal-6326.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6326.png,Normal/images/Normal-6326.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6326.png,Normal/masks/Normal-6326.png,True,False,299,299,L,167.71,a1dfebf1e7447fc0a2f499b126fc5a59
Normal-6327.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6327.png,Normal/images/Normal-6327.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6327.png,Normal/masks/Normal-6327.png,True,False,299,299,L,106.71,a7ae3f9e5d12b91706977f0a0bad5971
Normal-6328.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6328.png,Normal/images/Normal-6328.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6328.png,Normal/masks/Normal-6328.png,True,False,299,299,L,107.99,416539079f2c71c2a3ed60ab6b63c091
Normal-6329.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6329.png,Normal/images/Normal-6329.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6329.png,Normal/masks/Normal-6329.png,True,False,299,299,L,136.28,dcf3adc22d34387c3d1819b9cc3f2659
Normal-633.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-633.png,Normal/images/Normal-633.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-633.png,Normal/masks/Normal-633.png,True,False,299,299,L,115.7,c416c2b027b167970e774bed69ba4c9a
Normal-6330.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6330.png,Normal/images/Normal-6330.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6330.png,Normal/masks/Normal-6330.png,True,False,299,299,L,120.79,9fad63f2dd3717d8d81e85e6ddba1748
Normal-6331.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6331.png,Normal/images/Normal-6331.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6331.png,Normal/masks/Normal-6331.png,True,False,299,299,L,160.7,82addafa3662ee93535f9aa182235dd8
Normal-6332.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6332.png,Normal/images/Normal-6332.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6332.png,Normal/masks/Normal-6332.png,True,False,299,299,L,140.36,c7c1fc42cb19bf5d2f6102bbc00f85b4
Normal-6333.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6333.png,Normal/images/Normal-6333.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6333.png,Normal/masks/Normal-6333.png,True,False,299,299,L,124.22,bfad774aa09f6d9fd9b1327b41bf25f7
Normal-6334.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6334.png,Normal/images/Normal-6334.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6334.png,Normal/masks/Normal-6334.png,True,False,299,299,L,103.98,65b9593b975cadf42e69c8f0f49652ba
Normal-6335.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6335.png,Normal/images/Normal-6335.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6335.png,Normal/masks/Normal-6335.png,True,False,299,299,L,157.36,dd8391f3b8cc70e8605b0d5bee8c8df0
Normal-6336.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6336.png,Normal/images/Normal-6336.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6336.png,Normal/masks/Normal-6336.png,True,False,299,299,L,124.78,d54daccf6d52422c46b018878aeb1e83
Normal-6337.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6337.png,Normal/images/Normal-6337.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6337.png,Normal/masks/Normal-6337.png,True,False,299,299,L,103.45,95fe5bc446ac87a600eec0346998693c
Normal-6338.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6338.png,Normal/images/Normal-6338.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6338.png,Normal/masks/Normal-6338.png,True,False,299,299,L,125.2,cdc5e529893bf9c59c969b27a9f866de
Normal-6339.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6339.png,Normal/images/Normal-6339.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6339.png,Normal/masks/Normal-6339.png,True,False,299,299,L,132.67,18eb94be1f533cb978b9d3236195a082
Normal-634.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-634.png,Normal/images/Normal-634.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-634.png,Normal/masks/Normal-634.png,True,False,299,299,L,121.89,791d2d8adb0e4f15a813260cd0a25ad4
Normal-6340.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6340.png,Normal/images/Normal-6340.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6340.png,Normal/masks/Normal-6340.png,True,False,299,299,L,136.04,6f7e6577534d62108b096d07c220227f
Normal-6341.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6341.png,Normal/images/Normal-6341.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6341.png,Normal/masks/Normal-6341.png,True,False,299,299,L,78.74,5e72710ea31ab502d2011d29cb4ac0e2
Normal-6342.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6342.png,Normal/images/Normal-6342.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6342.png,Normal/masks/Normal-6342.png,True,False,299,299,L,117.07,6b1acfc65df5d3977c07bc589145c418
Normal-6343.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6343.png,Normal/images/Normal-6343.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6343.png,Normal/masks/Normal-6343.png,True,False,299,299,L,157.53,1fb1a12e6153e58d4887d734bde944cd
Normal-6344.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6344.png,Normal/images/Normal-6344.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6344.png,Normal/masks/Normal-6344.png,True,False,299,299,L,109.77,2e3a4f2927e3749bddade4adebdccb0e
Normal-6345.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6345.png,Normal/images/Normal-6345.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6345.png,Normal/masks/Normal-6345.png,True,False,299,299,L,90.9,5342af23b84cb5f7ba8bd3fc84965858
Normal-6346.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6346.png,Normal/images/Normal-6346.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6346.png,Normal/masks/Normal-6346.png,True,False,299,299,L,167.69,09fae52fff8e5e7fd84a6557fe62f525
Normal-6347.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6347.png,Normal/images/Normal-6347.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6347.png,Normal/masks/Normal-6347.png,True,False,299,299,L,113.68,2d7ea32fc37d8e35cecb39e539463c00
Normal-6348.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6348.png,Normal/images/Normal-6348.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6348.png,Normal/masks/Normal-6348.png,True,False,299,299,L,128.8,5625e9e03109e4d776c4d65151cb005b
Normal-6349.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6349.png,Normal/images/Normal-6349.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6349.png,Normal/masks/Normal-6349.png,True,False,299,299,L,108.87,3811cbfeb1ede0990f3d1ae9f31b08bc
Normal-635.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-635.png,Normal/images/Normal-635.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-635.png,Normal/masks/Normal-635.png,True,False,299,299,L,99.94,5d0ea6632c4aec869eae273f5808f216
Normal-6350.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6350.png,Normal/images/Normal-6350.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6350.png,Normal/masks/Normal-6350.png,True,False,299,299,L,173.77,db3e1d0473a148b03e2c678e45a065dc
Normal-6351.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6351.png,Normal/images/Normal-6351.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6351.png,Normal/masks/Normal-6351.png,True,False,299,299,L,130.91,50b965b47f2b438500e3ae4a69e707eb
Normal-6352.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6352.png,Normal/images/Normal-6352.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6352.png,Normal/masks/Normal-6352.png,True,False,299,299,L,111.67,6dcd7ccd152f6fde33726b363fc1ee12
Normal-6353.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6353.png,Normal/images/Normal-6353.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6353.png,Normal/masks/Normal-6353.png,True,False,299,299,L,119.35,8a88659c2f4669536a085bb50662b366
Normal-6354.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6354.png,Normal/images/Normal-6354.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6354.png,Normal/masks/Normal-6354.png,True,False,299,299,L,105.86,2e9a1c267ca76b8506d50a3e47a8352d
Normal-6355.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6355.png,Normal/images/Normal-6355.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6355.png,Normal/masks/Normal-6355.png,True,False,299,299,L,130.55,74d4f98c222702608f856f0b8ef67b6e
Normal-6356.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6356.png,Normal/images/Normal-6356.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6356.png,Normal/masks/Normal-6356.png,True,False,299,299,L,135.01,bef70024cc0b4871fbf0cc4866a78cef
Normal-6357.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6357.png,Normal/images/Normal-6357.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6357.png,Normal/masks/Normal-6357.png,True,False,299,299,L,154.23,f341fe5c9e6da446ce5e5254cdcf0160
Normal-6358.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6358.png,Normal/images/Normal-6358.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6358.png,Normal/masks/Normal-6358.png,True,False,299,299,L,102.56,c74042c198979b907a3547c0be53f1dd
Normal-6359.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6359.png,Normal/images/Normal-6359.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6359.png,Normal/masks/Normal-6359.png,True,False,299,299,L,102.68,546ab94a813164b4795a3744fd879826
Normal-636.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-636.png,Normal/images/Normal-636.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-636.png,Normal/masks/Normal-636.png,True,False,299,299,L,97.05,10686d5a5864b8e5f3a9451a6b208eee
Normal-6360.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6360.png,Normal/images/Normal-6360.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6360.png,Normal/masks/Normal-6360.png,True,False,299,299,L,139.5,c6cd1065af11cd82af1d3848d7708a9e
Normal-6361.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6361.png,Normal/images/Normal-6361.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6361.png,Normal/masks/Normal-6361.png,True,False,299,299,L,131.92,062f42aa91cd48ffba88c1eb7e5d84e0
Normal-6362.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6362.png,Normal/images/Normal-6362.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6362.png,Normal/masks/Normal-6362.png,True,False,299,299,L,121.12,3d43534497cd97db457c5ed280d6710d
Normal-6363.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6363.png,Normal/images/Normal-6363.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6363.png,Normal/masks/Normal-6363.png,True,False,299,299,L,144.92,ac434e0bd9dbbfc8e2c7a0e55236e1cc
Normal-6364.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6364.png,Normal/images/Normal-6364.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6364.png,Normal/masks/Normal-6364.png,True,False,299,299,L,111.98,a172a54d46a85f62ca3cc842953455a5
Normal-6365.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6365.png,Normal/images/Normal-6365.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6365.png,Normal/masks/Normal-6365.png,True,False,299,299,L,111.33,622ee84851a647612682ee180604d11d
Normal-6366.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6366.png,Normal/images/Normal-6366.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6366.png,Normal/masks/Normal-6366.png,True,False,299,299,L,78.14,ad496ce9163fa16dfab0c91be813b9c1
Normal-6367.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6367.png,Normal/images/Normal-6367.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6367.png,Normal/masks/Normal-6367.png,True,False,299,299,L,107.82,49ef161019bcadcd5412e21e58461955
Normal-6368.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6368.png,Normal/images/Normal-6368.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6368.png,Normal/masks/Normal-6368.png,True,False,299,299,L,109.63,8a9444c55f9743297c937886782ddd38
Normal-6369.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6369.png,Normal/images/Normal-6369.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6369.png,Normal/masks/Normal-6369.png,True,False,299,299,L,124.71,e6e48c7426a25bfdb37aad5c46f07a5d
Normal-637.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-637.png,Normal/images/Normal-637.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-637.png,Normal/masks/Normal-637.png,True,False,299,299,L,122.0,5c408a888aa9619a97eb271f72b52b57
Normal-6370.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6370.png,Normal/images/Normal-6370.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6370.png,Normal/masks/Normal-6370.png,True,False,299,299,L,107.83,ea21d44217d2bbce2953b7db8278477a
Normal-6371.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6371.png,Normal/images/Normal-6371.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6371.png,Normal/masks/Normal-6371.png,True,False,299,299,L,139.98,ac616ec14268ed9f529a7d1a3010e05a
Normal-6372.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6372.png,Normal/images/Normal-6372.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6372.png,Normal/masks/Normal-6372.png,True,False,299,299,L,103.72,fbff87a1ac47833f27ac3c5d742990bb
Normal-6373.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6373.png,Normal/images/Normal-6373.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6373.png,Normal/masks/Normal-6373.png,True,False,299,299,L,111.37,8932881a401b37806070d7cf5fcdeac2
Normal-6374.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6374.png,Normal/images/Normal-6374.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6374.png,Normal/masks/Normal-6374.png,True,False,299,299,L,74.03,bfb200f9b7cb818e9e2b0909bd3e4665
Normal-6375.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6375.png,Normal/images/Normal-6375.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6375.png,Normal/masks/Normal-6375.png,True,False,299,299,L,152.53,f4c287504fef19ad6b089ffa86453ccf
Normal-6376.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6376.png,Normal/images/Normal-6376.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6376.png,Normal/masks/Normal-6376.png,True,False,299,299,L,143.27,83aad0828efbe17181947131f143ecb5
Normal-6377.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6377.png,Normal/images/Normal-6377.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6377.png,Normal/masks/Normal-6377.png,True,False,299,299,L,106.84,3933135934483dc678db0b0a70c83ece
Normal-6378.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6378.png,Normal/images/Normal-6378.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6378.png,Normal/masks/Normal-6378.png,True,False,299,299,L,122.2,0f7fa3a94184c18e6ad39947066ea678
Normal-6379.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6379.png,Normal/images/Normal-6379.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6379.png,Normal/masks/Normal-6379.png,True,False,299,299,L,113.91,230abf1af3b61427c0746889af668b24
Normal-638.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-638.png,Normal/images/Normal-638.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-638.png,Normal/masks/Normal-638.png,True,False,299,299,L,120.95,82373ba8095447c96ff13b7a3d798e11
Normal-6380.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6380.png,Normal/images/Normal-6380.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6380.png,Normal/masks/Normal-6380.png,True,False,299,299,L,136.94,c62bb1e890d4ccb17e5b1beae581283a
Normal-6381.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6381.png,Normal/images/Normal-6381.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6381.png,Normal/masks/Normal-6381.png,True,False,299,299,L,143.26,6137e8746b36dee5b9b7b13918da2030
Normal-6382.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6382.png,Normal/images/Normal-6382.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6382.png,Normal/masks/Normal-6382.png,True,False,299,299,L,160.28,08c24bbea56fa53eb9aeb973435d0575
Normal-6383.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6383.png,Normal/images/Normal-6383.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6383.png,Normal/masks/Normal-6383.png,True,False,299,299,L,95.73,1cac1097193f1e0c51ab2c2ad4fb3b41
Normal-6384.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6384.png,Normal/images/Normal-6384.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6384.png,Normal/masks/Normal-6384.png,True,False,299,299,L,168.5,f1e889e2684411c78ed28c0f8a6518df
Normal-6385.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6385.png,Normal/images/Normal-6385.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6385.png,Normal/masks/Normal-6385.png,True,False,299,299,L,135.55,2a0e40ebd37dcde66b4bc977a0f2dfbc
Normal-6386.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6386.png,Normal/images/Normal-6386.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6386.png,Normal/masks/Normal-6386.png,True,False,299,299,L,122.2,fa684ef3eb7028b0d4beeb512a2e4da7
Normal-6387.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6387.png,Normal/images/Normal-6387.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6387.png,Normal/masks/Normal-6387.png,True,False,299,299,L,119.31,7cbe65313956fb23e12df30df438d128
Normal-6388.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6388.png,Normal/images/Normal-6388.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6388.png,Normal/masks/Normal-6388.png,True,False,299,299,L,105.72,119205c4662c01b395168d785dde77b8
Normal-6389.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6389.png,Normal/images/Normal-6389.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6389.png,Normal/masks/Normal-6389.png,True,False,299,299,L,123.93,52f2e60f588d9bc4e814cf2242c90d92
Normal-639.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-639.png,Normal/images/Normal-639.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-639.png,Normal/masks/Normal-639.png,True,False,299,299,L,110.73,29cf460100fc02ac08d8e060fe2bd487
Normal-6390.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6390.png,Normal/images/Normal-6390.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6390.png,Normal/masks/Normal-6390.png,True,False,299,299,L,108.42,d5d14db5acc2d1d6076cc675af12a997
Normal-6391.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6391.png,Normal/images/Normal-6391.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6391.png,Normal/masks/Normal-6391.png,True,False,299,299,L,128.24,3c8391a8496c420902bea84eaa895d0d
Normal-6392.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6392.png,Normal/images/Normal-6392.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6392.png,Normal/masks/Normal-6392.png,True,False,299,299,L,125.66,d5c5960450160b62d20c26169d31e3e3
Normal-6393.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6393.png,Normal/images/Normal-6393.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6393.png,Normal/masks/Normal-6393.png,True,False,299,299,L,135.72,5a5cc784cfcf794e65801c98811b0462
Normal-6394.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6394.png,Normal/images/Normal-6394.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6394.png,Normal/masks/Normal-6394.png,True,False,299,299,L,99.26,deb9c1d62e0b1e5d7ca0daabe3b81d0b
Normal-6395.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6395.png,Normal/images/Normal-6395.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6395.png,Normal/masks/Normal-6395.png,True,False,299,299,L,94.79,02961fc293a8f32da790a4e64164bbdf
Normal-6396.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6396.png,Normal/images/Normal-6396.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6396.png,Normal/masks/Normal-6396.png,True,False,299,299,L,103.74,1639c909dab528cbdc8d56e1773f0b7c
Normal-6397.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6397.png,Normal/images/Normal-6397.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6397.png,Normal/masks/Normal-6397.png,True,False,299,299,L,121.55,5ba9959b347c3dceef26ff26c57b21a6
Normal-6398.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6398.png,Normal/images/Normal-6398.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6398.png,Normal/masks/Normal-6398.png,True,False,299,299,L,169.09,a05ad941815e4bf7dfb835fa22190e16
Normal-6399.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6399.png,Normal/images/Normal-6399.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6399.png,Normal/masks/Normal-6399.png,True,False,299,299,L,116.56,a46157449a318c710e45538b82fa5ccd
Normal-64.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-64.png,Normal/images/Normal-64.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-64.png,Normal/masks/Normal-64.png,True,False,299,299,L,100.65,92928067430258dad1b6d5c449c54fa3
Normal-640.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-640.png,Normal/images/Normal-640.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-640.png,Normal/masks/Normal-640.png,True,False,299,299,L,123.82,cf76f0ce26f639651e2107b546d4617a
Normal-6400.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6400.png,Normal/images/Normal-6400.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6400.png,Normal/masks/Normal-6400.png,True,False,299,299,L,139.95,bbabd9ead7fec04d9aabbe2f10fef584
Normal-6401.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6401.png,Normal/images/Normal-6401.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6401.png,Normal/masks/Normal-6401.png,True,False,299,299,L,149.72,a68bb5e4332836898ed9c3ced7dc2f96
Normal-6402.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6402.png,Normal/images/Normal-6402.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6402.png,Normal/masks/Normal-6402.png,True,False,299,299,L,106.22,a2b87b8b91baf182c6a3f4c211a5961a
Normal-6403.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6403.png,Normal/images/Normal-6403.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6403.png,Normal/masks/Normal-6403.png,True,False,299,299,L,161.72,cc424f06eca12e41e713455b2adebdb6
Normal-6404.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6404.png,Normal/images/Normal-6404.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6404.png,Normal/masks/Normal-6404.png,True,False,299,299,L,165.38,165529d082de0d66506f6e189e19cda9
Normal-6405.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6405.png,Normal/images/Normal-6405.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6405.png,Normal/masks/Normal-6405.png,True,False,299,299,L,107.65,e7cde0d8a73e6c560856bcc9effa5d25
Normal-6406.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6406.png,Normal/images/Normal-6406.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6406.png,Normal/masks/Normal-6406.png,True,False,299,299,L,170.08,131c01497e15b7c8d1ba599e16a514a5
Normal-6407.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6407.png,Normal/images/Normal-6407.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6407.png,Normal/masks/Normal-6407.png,True,False,299,299,L,129.0,cbdfd195dcba3ceb1c48171728795700
Normal-6408.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6408.png,Normal/images/Normal-6408.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6408.png,Normal/masks/Normal-6408.png,True,False,299,299,L,110.08,00076eb127acb0bdd4d989e177588c1e
Normal-6409.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6409.png,Normal/images/Normal-6409.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6409.png,Normal/masks/Normal-6409.png,True,False,299,299,L,109.76,33f84326972836a20d9efe35eb1637f8
Normal-641.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-641.png,Normal/images/Normal-641.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-641.png,Normal/masks/Normal-641.png,True,False,299,299,L,127.49,ea8ad4de152e6f01208e4100de3ad0f8
Normal-6410.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6410.png,Normal/images/Normal-6410.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6410.png,Normal/masks/Normal-6410.png,True,False,299,299,L,178.13,6961f23d3e4a32812d684a93b48956fd
Normal-6411.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6411.png,Normal/images/Normal-6411.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6411.png,Normal/masks/Normal-6411.png,True,False,299,299,L,120.02,9e2d9f37b73eb1e113e97df7cd55fe1d
Normal-6412.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6412.png,Normal/images/Normal-6412.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6412.png,Normal/masks/Normal-6412.png,True,False,299,299,L,164.99,c0da442bf9d1b0f61b1c7b37921a98c0
Normal-6413.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6413.png,Normal/images/Normal-6413.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6413.png,Normal/masks/Normal-6413.png,True,False,299,299,L,104.08,b3bb49d99486a3fc1971d4f8c7d17d1e
Normal-6414.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6414.png,Normal/images/Normal-6414.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6414.png,Normal/masks/Normal-6414.png,True,False,299,299,L,126.86,53fbf7c665c09ccfbd59b854863227a4
Normal-6415.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6415.png,Normal/images/Normal-6415.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6415.png,Normal/masks/Normal-6415.png,True,False,299,299,L,129.99,646c6ddc19923ec39d3b70b773eac37f
Normal-6416.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6416.png,Normal/images/Normal-6416.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6416.png,Normal/masks/Normal-6416.png,True,False,299,299,L,121.02,ed269da27dfb09c32f407d5b4c0d9009
Normal-6417.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6417.png,Normal/images/Normal-6417.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6417.png,Normal/masks/Normal-6417.png,True,False,299,299,L,126.76,468000f6e76b9e9b0b3ea9bb4b31c127
Normal-6418.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6418.png,Normal/images/Normal-6418.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6418.png,Normal/masks/Normal-6418.png,True,False,299,299,L,130.53,02c7174e3cab942dd30312435aec9354
Normal-6419.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6419.png,Normal/images/Normal-6419.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6419.png,Normal/masks/Normal-6419.png,True,False,299,299,L,110.87,7f88d836f7b61f0f9fc5f8338f3edbbc
Normal-642.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-642.png,Normal/images/Normal-642.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-642.png,Normal/masks/Normal-642.png,True,False,299,299,L,116.75,d55e11242afdfa8047a4530259de7b49
Normal-6420.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6420.png,Normal/images/Normal-6420.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6420.png,Normal/masks/Normal-6420.png,True,False,299,299,L,117.25,b2bec683a2c125cf704c8d2066245bf6
Normal-6421.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6421.png,Normal/images/Normal-6421.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6421.png,Normal/masks/Normal-6421.png,True,False,299,299,L,176.86,9a975f2537ade4c4807b78098e88904d
Normal-6422.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6422.png,Normal/images/Normal-6422.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6422.png,Normal/masks/Normal-6422.png,True,False,299,299,L,112.92,0bbff3c08ca5e60d6218df76c8302a28
Normal-6423.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6423.png,Normal/images/Normal-6423.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6423.png,Normal/masks/Normal-6423.png,True,False,299,299,L,160.09,daf69b9ba724d96a8d4d1e9bccaa7a2d
Normal-6424.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6424.png,Normal/images/Normal-6424.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6424.png,Normal/masks/Normal-6424.png,True,False,299,299,L,168.44,77d62d4ed1b1c18ac22d9ddcec51268b
Normal-6425.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6425.png,Normal/images/Normal-6425.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6425.png,Normal/masks/Normal-6425.png,True,False,299,299,L,94.98,553d554aa406811dca5c729523b284c7
Normal-6426.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6426.png,Normal/images/Normal-6426.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6426.png,Normal/masks/Normal-6426.png,True,False,299,299,L,98.97,e5e7085754a4be2f971df01123d1e4d4
Normal-6427.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6427.png,Normal/images/Normal-6427.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6427.png,Normal/masks/Normal-6427.png,True,False,299,299,L,169.86,418e0ef8fb68f47747bdd585387d4ecf
Normal-6428.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6428.png,Normal/images/Normal-6428.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6428.png,Normal/masks/Normal-6428.png,True,False,299,299,L,142.06,17e96a7e89904a41bcb560b58418f826
Normal-6429.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6429.png,Normal/images/Normal-6429.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6429.png,Normal/masks/Normal-6429.png,True,False,299,299,L,160.47,e3e1184968fecc0cbeb0c6262697e060
Normal-643.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-643.png,Normal/images/Normal-643.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-643.png,Normal/masks/Normal-643.png,True,False,299,299,L,116.82,63cbb599bb318cd522667f6dafbb8248
Normal-6430.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6430.png,Normal/images/Normal-6430.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6430.png,Normal/masks/Normal-6430.png,True,False,299,299,L,99.71,e3e456dd8c1ba64a85f668450164d329
Normal-6431.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6431.png,Normal/images/Normal-6431.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6431.png,Normal/masks/Normal-6431.png,True,False,299,299,L,114.93,03028777c0de33cf6d3365dd86bf3a65
Normal-6432.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6432.png,Normal/images/Normal-6432.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6432.png,Normal/masks/Normal-6432.png,True,False,299,299,L,161.24,3d994f3d50e31832ae9a6b8268d972f5
Normal-6433.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6433.png,Normal/images/Normal-6433.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6433.png,Normal/masks/Normal-6433.png,True,False,299,299,L,111.09,2012a70afaec496d0f534680f1153663
Normal-6434.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6434.png,Normal/images/Normal-6434.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6434.png,Normal/masks/Normal-6434.png,True,False,299,299,L,141.37,2f4b54c10ec747d0fd739e8c93bc64f6
Normal-6435.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6435.png,Normal/images/Normal-6435.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6435.png,Normal/masks/Normal-6435.png,True,False,299,299,L,120.83,20dbd9251379cd407040d73a5ce0655c
Normal-6436.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6436.png,Normal/images/Normal-6436.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6436.png,Normal/masks/Normal-6436.png,True,False,299,299,L,119.72,e459adae1866ca8ac4ca2f19f2d92a2f
Normal-6437.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6437.png,Normal/images/Normal-6437.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6437.png,Normal/masks/Normal-6437.png,True,False,299,299,L,139.8,6ad81e5a2776e4de915ff8fcf095086e
Normal-6438.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6438.png,Normal/images/Normal-6438.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6438.png,Normal/masks/Normal-6438.png,True,False,299,299,L,138.73,8bb8fcd8af14b4487e0d89c45c9b1f23
Normal-6439.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6439.png,Normal/images/Normal-6439.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6439.png,Normal/masks/Normal-6439.png,True,False,299,299,L,130.16,b63acd32a9a836df4f28f0a8b5318587
Normal-644.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-644.png,Normal/images/Normal-644.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-644.png,Normal/masks/Normal-644.png,True,False,299,299,L,99.81,d7cec525fdf4806d413c8049816956cc
Normal-6440.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6440.png,Normal/images/Normal-6440.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6440.png,Normal/masks/Normal-6440.png,True,False,299,299,L,172.87,6ad66eefb2955f9eec91d52980d21544
Normal-6441.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6441.png,Normal/images/Normal-6441.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6441.png,Normal/masks/Normal-6441.png,True,False,299,299,L,109.42,ed56d07d9bf9d7ceed7deda4085e598e
Normal-6442.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6442.png,Normal/images/Normal-6442.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6442.png,Normal/masks/Normal-6442.png,True,False,299,299,L,122.76,af45a263e5177de2f7e7a5f7771f5107
Normal-6443.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6443.png,Normal/images/Normal-6443.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6443.png,Normal/masks/Normal-6443.png,True,False,299,299,L,176.1,56acd0a0a8374917f5b5981015e82e1c
Normal-6444.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6444.png,Normal/images/Normal-6444.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6444.png,Normal/masks/Normal-6444.png,True,False,299,299,L,160.78,51b881fd97a02521b148471080f262dc
Normal-6445.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6445.png,Normal/images/Normal-6445.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6445.png,Normal/masks/Normal-6445.png,True,False,299,299,L,102.58,6fdced2ef27c15862bf5ebb207e9fea9
Normal-6446.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6446.png,Normal/images/Normal-6446.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6446.png,Normal/masks/Normal-6446.png,True,False,299,299,L,138.76,d011137deb6016a56cc0ab5700c1e994
Normal-6447.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6447.png,Normal/images/Normal-6447.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6447.png,Normal/masks/Normal-6447.png,True,False,299,299,L,112.14,0b19da4a368aa000aafbce8f66bf8f85
Normal-6448.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6448.png,Normal/images/Normal-6448.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6448.png,Normal/masks/Normal-6448.png,True,False,299,299,L,135.5,6ab92128081b50882cdf7fb8c317f33f
Normal-6449.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6449.png,Normal/images/Normal-6449.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6449.png,Normal/masks/Normal-6449.png,True,False,299,299,L,139.17,54a032b0702ea8b2766568db8b3f9d84
Normal-645.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-645.png,Normal/images/Normal-645.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-645.png,Normal/masks/Normal-645.png,True,False,299,299,L,96.06,82166b741b23a2c72c81759b4ad4f4e4
Normal-6450.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6450.png,Normal/images/Normal-6450.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6450.png,Normal/masks/Normal-6450.png,True,False,299,299,L,116.3,f1086cf1008b585db2ab9624d068f061
Normal-6451.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6451.png,Normal/images/Normal-6451.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6451.png,Normal/masks/Normal-6451.png,True,False,299,299,L,113.28,acaf38265231b513f6939afc14064be8
Normal-6452.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6452.png,Normal/images/Normal-6452.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6452.png,Normal/masks/Normal-6452.png,True,False,299,299,L,136.76,d13f40d26a47fc472a42b24b6cdfe3ff
Normal-6453.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6453.png,Normal/images/Normal-6453.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6453.png,Normal/masks/Normal-6453.png,True,False,299,299,L,148.53,58afdb26b042fc832bf86a8509d205da
Normal-6454.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6454.png,Normal/images/Normal-6454.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6454.png,Normal/masks/Normal-6454.png,True,False,299,299,L,136.31,ff62adb597f8fb42dad601af56d16ca6
Normal-6455.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6455.png,Normal/images/Normal-6455.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6455.png,Normal/masks/Normal-6455.png,True,False,299,299,L,116.24,88585da1d0b08650630a400453aa3064
Normal-6456.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6456.png,Normal/images/Normal-6456.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6456.png,Normal/masks/Normal-6456.png,True,False,299,299,L,136.2,6d99567ccaaa766f2244d4ebc100ff13
Normal-6457.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6457.png,Normal/images/Normal-6457.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6457.png,Normal/masks/Normal-6457.png,True,False,299,299,L,131.42,a5a2ee583856791615d5f0c1e1c59d4d
Normal-6458.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6458.png,Normal/images/Normal-6458.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6458.png,Normal/masks/Normal-6458.png,True,False,299,299,L,138.66,9ee1cd9b5d19a07321457f6c154add0b
Normal-6459.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6459.png,Normal/images/Normal-6459.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6459.png,Normal/masks/Normal-6459.png,True,False,299,299,L,140.61,fec523045202e0f22058d7bf785a54e8
Normal-646.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-646.png,Normal/images/Normal-646.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-646.png,Normal/masks/Normal-646.png,True,False,299,299,L,97.99,deaf77a7f964aebe8073277c8fe227d8
Normal-6460.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6460.png,Normal/images/Normal-6460.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6460.png,Normal/masks/Normal-6460.png,True,False,299,299,L,111.62,86a358e8052a1c2202bfe37eb8e3394f
Normal-6461.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6461.png,Normal/images/Normal-6461.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6461.png,Normal/masks/Normal-6461.png,True,False,299,299,L,123.02,a3e12410137e220599a86beb1c89ba7b
Normal-6462.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6462.png,Normal/images/Normal-6462.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6462.png,Normal/masks/Normal-6462.png,True,False,299,299,L,106.11,df480b58e6e476de3a1c79d6d8d84070
Normal-6463.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6463.png,Normal/images/Normal-6463.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6463.png,Normal/masks/Normal-6463.png,True,False,299,299,L,106.39,9838639b49a0f3f7455aa3611ac5bb1b
Normal-6464.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6464.png,Normal/images/Normal-6464.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6464.png,Normal/masks/Normal-6464.png,True,False,299,299,L,164.15,cfaf9185a74cd72b8fa77d2e8969ca37
Normal-6465.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6465.png,Normal/images/Normal-6465.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6465.png,Normal/masks/Normal-6465.png,True,False,299,299,L,145.87,43b7101587ab2b42513682fdfa7000e2
Normal-6466.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6466.png,Normal/images/Normal-6466.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6466.png,Normal/masks/Normal-6466.png,True,False,299,299,L,104.08,c9c93b772de7ffe09ef5e5e3ea4c1ca6
Normal-6467.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6467.png,Normal/images/Normal-6467.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6467.png,Normal/masks/Normal-6467.png,True,False,299,299,L,122.91,2361b9b194acc2c76718ebba7a6f15a3
Normal-6468.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6468.png,Normal/images/Normal-6468.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6468.png,Normal/masks/Normal-6468.png,True,False,299,299,L,133.3,3d94dc13802644d3d6d094cb92ede080
Normal-6469.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6469.png,Normal/images/Normal-6469.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6469.png,Normal/masks/Normal-6469.png,True,False,299,299,L,126.02,e9b502cf08648f9329578d050b069e53
Normal-647.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-647.png,Normal/images/Normal-647.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-647.png,Normal/masks/Normal-647.png,True,False,299,299,L,117.95,c01276ffde181b817e265c0631a383f7
Normal-6470.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6470.png,Normal/images/Normal-6470.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6470.png,Normal/masks/Normal-6470.png,True,False,299,299,L,116.03,4ea1ef379110be1adf3a177954992bb3
Normal-6471.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6471.png,Normal/images/Normal-6471.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6471.png,Normal/masks/Normal-6471.png,True,False,299,299,L,113.07,7f4661ec9f56f09fba561f019040c960
Normal-6472.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6472.png,Normal/images/Normal-6472.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6472.png,Normal/masks/Normal-6472.png,True,False,299,299,L,155.2,f7d7678ef51eeb2a697a718c1ba4783a
Normal-6473.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6473.png,Normal/images/Normal-6473.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6473.png,Normal/masks/Normal-6473.png,True,False,299,299,L,109.88,a18155cc55ba414bd5b1491eb73d47e7
Normal-6474.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6474.png,Normal/images/Normal-6474.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6474.png,Normal/masks/Normal-6474.png,True,False,299,299,L,98.8,5b4a8e1857f7d69b2860632cf6696551
Normal-6475.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6475.png,Normal/images/Normal-6475.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6475.png,Normal/masks/Normal-6475.png,True,False,299,299,L,149.69,3b0a4b88ddb833047b713bfa57c0ed1e
Normal-6476.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6476.png,Normal/images/Normal-6476.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6476.png,Normal/masks/Normal-6476.png,True,False,299,299,L,129.01,232c47612524d638ab5974f6a9f19712
Normal-6477.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6477.png,Normal/images/Normal-6477.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6477.png,Normal/masks/Normal-6477.png,True,False,299,299,L,112.32,a03d4544b95d6658c684f8a122bfe94b
Normal-6478.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6478.png,Normal/images/Normal-6478.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6478.png,Normal/masks/Normal-6478.png,True,False,299,299,L,124.47,0d48a4d682af9ba4870312350f7fcbaa
Normal-6479.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6479.png,Normal/images/Normal-6479.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6479.png,Normal/masks/Normal-6479.png,True,False,299,299,L,112.21,f66d103a3d02a1b60ffeff484b22c647
Normal-648.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-648.png,Normal/images/Normal-648.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-648.png,Normal/masks/Normal-648.png,True,False,299,299,L,113.66,d13f8c5ff68febe77bdb0139afe4c088
Normal-6480.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6480.png,Normal/images/Normal-6480.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6480.png,Normal/masks/Normal-6480.png,True,False,299,299,L,162.41,d6622182819cdfabb001c1f4a70a43f6
Normal-6481.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6481.png,Normal/images/Normal-6481.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6481.png,Normal/masks/Normal-6481.png,True,False,299,299,L,109.27,6059ca5b2d9b7b4fc033f232517f2775
Normal-6482.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6482.png,Normal/images/Normal-6482.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6482.png,Normal/masks/Normal-6482.png,True,False,299,299,L,172.28,71145175d522b873639044d7ed245b7f
Normal-6483.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6483.png,Normal/images/Normal-6483.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6483.png,Normal/masks/Normal-6483.png,True,False,299,299,L,155.32,bb3f1a07a6c00ca5778381a652cca9cc
Normal-6484.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6484.png,Normal/images/Normal-6484.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6484.png,Normal/masks/Normal-6484.png,True,False,299,299,L,105.92,0063f070d186200478baa5bcf313ac01
Normal-6485.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6485.png,Normal/images/Normal-6485.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6485.png,Normal/masks/Normal-6485.png,True,False,299,299,L,59.1,01c753c95a8c6b923c8eab354645d700
Normal-6486.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6486.png,Normal/images/Normal-6486.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6486.png,Normal/masks/Normal-6486.png,True,False,299,299,L,109.79,fe7ab9b6d464ba10a82df5569e3478fd
Normal-6487.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6487.png,Normal/images/Normal-6487.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6487.png,Normal/masks/Normal-6487.png,True,False,299,299,L,128.37,196f9b5314eea0a0773bda8e7c1368ab
Normal-6488.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6488.png,Normal/images/Normal-6488.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6488.png,Normal/masks/Normal-6488.png,True,False,299,299,L,103.42,20533e81bb3ff0229d857d3f25701e6c
Normal-6489.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6489.png,Normal/images/Normal-6489.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6489.png,Normal/masks/Normal-6489.png,True,False,299,299,L,157.18,4026e69d996c96099f0422c56bc59d7c
Normal-649.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-649.png,Normal/images/Normal-649.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-649.png,Normal/masks/Normal-649.png,True,False,299,299,L,129.87,e27dcd884aa20fc44b66b1e587219f26
Normal-6490.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6490.png,Normal/images/Normal-6490.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6490.png,Normal/masks/Normal-6490.png,True,False,299,299,L,130.38,8c353c071433003f10f0f72feba48145
Normal-6491.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6491.png,Normal/images/Normal-6491.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6491.png,Normal/masks/Normal-6491.png,True,False,299,299,L,109.62,7316c03ae4f3ff43b725aa5158a07769
Normal-6492.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6492.png,Normal/images/Normal-6492.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6492.png,Normal/masks/Normal-6492.png,True,False,299,299,L,116.0,0c3318fe98204b0a24ee02b444497526
Normal-6493.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6493.png,Normal/images/Normal-6493.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6493.png,Normal/masks/Normal-6493.png,True,False,299,299,L,105.3,859d585439da2f813fd81425544d6486
Normal-6494.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6494.png,Normal/images/Normal-6494.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6494.png,Normal/masks/Normal-6494.png,True,False,299,299,L,138.45,163ce93c64e56f203049fb0e7968fd07
Normal-6495.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6495.png,Normal/images/Normal-6495.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6495.png,Normal/masks/Normal-6495.png,True,False,299,299,L,108.81,0861da8d95267e61cc3867df46bdc34d
Normal-6496.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6496.png,Normal/images/Normal-6496.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6496.png,Normal/masks/Normal-6496.png,True,False,299,299,L,116.3,e2df152f30cce70afe84ee8f3079f930
Normal-6497.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6497.png,Normal/images/Normal-6497.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6497.png,Normal/masks/Normal-6497.png,True,False,299,299,L,147.26,feae5f0c13ac3cb88ab4cfd42e5362fe
Normal-6498.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6498.png,Normal/images/Normal-6498.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6498.png,Normal/masks/Normal-6498.png,True,False,299,299,L,104.71,1e53c91c132c1d6b42f67c7e4c13cea1
Normal-6499.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6499.png,Normal/images/Normal-6499.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6499.png,Normal/masks/Normal-6499.png,True,False,299,299,L,144.07,4b75151b3b2a363bb947119863cb4592
Normal-65.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-65.png,Normal/images/Normal-65.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-65.png,Normal/masks/Normal-65.png,True,False,299,299,L,121.98,43d6e0a7c7dbebf469b6a31071ccf18b
Normal-650.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-650.png,Normal/images/Normal-650.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-650.png,Normal/masks/Normal-650.png,True,False,299,299,L,96.05,4f00153297b3ad7d47af8f98731dd17b
Normal-6500.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6500.png,Normal/images/Normal-6500.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6500.png,Normal/masks/Normal-6500.png,True,False,299,299,L,124.04,a1c2849a3dd0d70f96c680195de7f197
Normal-6501.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6501.png,Normal/images/Normal-6501.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6501.png,Normal/masks/Normal-6501.png,True,False,299,299,L,111.93,926319f5485fb917012eee634c8bd1d4
Normal-6502.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6502.png,Normal/images/Normal-6502.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6502.png,Normal/masks/Normal-6502.png,True,False,299,299,L,101.63,57e65f10cbb04290ad499e7d0cffbf98
Normal-6503.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6503.png,Normal/images/Normal-6503.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6503.png,Normal/masks/Normal-6503.png,True,False,299,299,L,121.56,0328c386f847e57d955d1f7f2f6a550d
Normal-6504.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6504.png,Normal/images/Normal-6504.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6504.png,Normal/masks/Normal-6504.png,True,False,299,299,L,110.29,fb98c00be0f6032f7aad677f45e174cd
Normal-6505.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6505.png,Normal/images/Normal-6505.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6505.png,Normal/masks/Normal-6505.png,True,False,299,299,L,109.4,26844c7f2e152a19fc948577532963c5
Normal-6506.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6506.png,Normal/images/Normal-6506.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6506.png,Normal/masks/Normal-6506.png,True,False,299,299,L,175.88,c8158f64deb1c50158a24b8c41bdcc1c
Normal-6507.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6507.png,Normal/images/Normal-6507.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6507.png,Normal/masks/Normal-6507.png,True,False,299,299,L,158.98,bb5202b29e0e012c4cb5f829b4536789
Normal-6508.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6508.png,Normal/images/Normal-6508.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6508.png,Normal/masks/Normal-6508.png,True,False,299,299,L,151.27,01a339327b2d4d17718ead24cd91b9c9
Normal-6509.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6509.png,Normal/images/Normal-6509.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6509.png,Normal/masks/Normal-6509.png,True,False,299,299,L,83.91,5e07431dd51fd8e59b541cd29a886ea0
Normal-651.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-651.png,Normal/images/Normal-651.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-651.png,Normal/masks/Normal-651.png,True,False,299,299,L,119.12,a5f299491f3bc0020b0f5e8df7097d8f
Normal-6510.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6510.png,Normal/images/Normal-6510.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6510.png,Normal/masks/Normal-6510.png,True,False,299,299,L,160.75,8e0149f66bdf7820bc94efd636601b41
Normal-6511.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6511.png,Normal/images/Normal-6511.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6511.png,Normal/masks/Normal-6511.png,True,False,299,299,L,163.84,10604407cf49a799690538ab3d45ebbf
Normal-6512.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6512.png,Normal/images/Normal-6512.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6512.png,Normal/masks/Normal-6512.png,True,False,299,299,L,103.99,40042be6de3ce477712a9951ababe1cd
Normal-6513.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6513.png,Normal/images/Normal-6513.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6513.png,Normal/masks/Normal-6513.png,True,False,299,299,L,117.13,33f933d6961d738071d6f7e341c96dfb
Normal-6514.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6514.png,Normal/images/Normal-6514.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6514.png,Normal/masks/Normal-6514.png,True,False,299,299,L,129.31,9b70e4b9cc94beed1703a066dab52111
Normal-6515.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6515.png,Normal/images/Normal-6515.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6515.png,Normal/masks/Normal-6515.png,True,False,299,299,L,123.55,d1591cdaeec006ecf362dc9d3529c05f
Normal-6516.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6516.png,Normal/images/Normal-6516.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6516.png,Normal/masks/Normal-6516.png,True,False,299,299,L,115.11,bd781766efb9c008653b71ac199b4410
Normal-6517.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6517.png,Normal/images/Normal-6517.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6517.png,Normal/masks/Normal-6517.png,True,False,299,299,L,111.16,2e81dd42f7030de47c5079b65528eb34
Normal-6518.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6518.png,Normal/images/Normal-6518.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6518.png,Normal/masks/Normal-6518.png,True,False,299,299,L,153.92,c1a29d8d736cf59be44a550d2db83fb0
Normal-6519.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6519.png,Normal/images/Normal-6519.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6519.png,Normal/masks/Normal-6519.png,True,False,299,299,L,171.3,eb65cdc85a2f0eb644f22813e58b6268
Normal-652.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-652.png,Normal/images/Normal-652.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-652.png,Normal/masks/Normal-652.png,True,False,299,299,L,121.9,a62ed7eb97c116438df977ccf89c48b4
Normal-6520.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6520.png,Normal/images/Normal-6520.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6520.png,Normal/masks/Normal-6520.png,True,False,299,299,L,107.01,0357cab6da2acf107d6efc2e05dbb604
Normal-6521.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6521.png,Normal/images/Normal-6521.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6521.png,Normal/masks/Normal-6521.png,True,False,299,299,L,95.65,61d5be6ca54260c2a50c40491381596e
Normal-6522.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6522.png,Normal/images/Normal-6522.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6522.png,Normal/masks/Normal-6522.png,True,False,299,299,L,148.43,e9a38e0aac0505178dd9148a0af9e2f8
Normal-6523.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6523.png,Normal/images/Normal-6523.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6523.png,Normal/masks/Normal-6523.png,True,False,299,299,L,142.03,c8b423c2ae34b3209820f1e3a98c525e
Normal-6524.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6524.png,Normal/images/Normal-6524.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6524.png,Normal/masks/Normal-6524.png,True,False,299,299,L,130.16,87070f54783939aedf5648f8601a9e95
Normal-6525.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6525.png,Normal/images/Normal-6525.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6525.png,Normal/masks/Normal-6525.png,True,False,299,299,L,167.35,3ebe9986479b51f8f0651c58a6ccd4f4
Normal-6526.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6526.png,Normal/images/Normal-6526.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6526.png,Normal/masks/Normal-6526.png,True,False,299,299,L,139.03,8e7d010663c0dc29e73d84dd6b2fe777
Normal-6527.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6527.png,Normal/images/Normal-6527.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6527.png,Normal/masks/Normal-6527.png,True,False,299,299,L,163.53,7c9c7ba0e885461ff1c8307f22e4006f
Normal-6528.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6528.png,Normal/images/Normal-6528.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6528.png,Normal/masks/Normal-6528.png,True,False,299,299,L,122.37,6ff1d22f287a910e04d446874ad81624
Normal-6529.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6529.png,Normal/images/Normal-6529.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6529.png,Normal/masks/Normal-6529.png,True,False,299,299,L,155.69,2f12ad1e53d534d73805a19597fdd0f4
Normal-653.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-653.png,Normal/images/Normal-653.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-653.png,Normal/masks/Normal-653.png,True,False,299,299,L,117.74,1495e6fd1e4c62a3566560e8ae1dfc39
Normal-6530.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6530.png,Normal/images/Normal-6530.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6530.png,Normal/masks/Normal-6530.png,True,False,299,299,L,125.16,230de62ff2a6d25b04984bdc3a8c64ac
Normal-6531.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6531.png,Normal/images/Normal-6531.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6531.png,Normal/masks/Normal-6531.png,True,False,299,299,L,112.76,4283cdf477dbc7c8d0e20300a5292d3a
Normal-6532.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6532.png,Normal/images/Normal-6532.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6532.png,Normal/masks/Normal-6532.png,True,False,299,299,L,160.82,579ee69f4e34c6a5407b6d012ea8c098
Normal-6533.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6533.png,Normal/images/Normal-6533.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6533.png,Normal/masks/Normal-6533.png,True,False,299,299,L,123.26,ea62b5a6ce7513423b10c64b0cd0cfd7
Normal-6534.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6534.png,Normal/images/Normal-6534.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6534.png,Normal/masks/Normal-6534.png,True,False,299,299,L,158.13,55211d1a79e83a6063fca26e145bb798
Normal-6535.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6535.png,Normal/images/Normal-6535.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6535.png,Normal/masks/Normal-6535.png,True,False,299,299,L,133.12,37ca0ad3627c7fdbc0b815da249eba78
Normal-6536.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6536.png,Normal/images/Normal-6536.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6536.png,Normal/masks/Normal-6536.png,True,False,299,299,L,142.42,8236cc91175f4d304ad1529dab690321
Normal-6537.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6537.png,Normal/images/Normal-6537.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6537.png,Normal/masks/Normal-6537.png,True,False,299,299,L,107.14,e295ec11854c9e2da7d8b1691b6add34
Normal-6538.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6538.png,Normal/images/Normal-6538.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6538.png,Normal/masks/Normal-6538.png,True,False,299,299,L,140.17,a4551c12bcf3b283d5b26a34791f42ca
Normal-6539.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6539.png,Normal/images/Normal-6539.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6539.png,Normal/masks/Normal-6539.png,True,False,299,299,L,127.7,7b92b1109f7a35f3ec153f1a4dfc20da
Normal-654.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-654.png,Normal/images/Normal-654.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-654.png,Normal/masks/Normal-654.png,True,False,299,299,L,125.16,1c0dcc2d90b7929b9417aae5033590f2
Normal-6540.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6540.png,Normal/images/Normal-6540.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6540.png,Normal/masks/Normal-6540.png,True,False,299,299,L,108.32,4a4f008cf7ec31565707b6920d9ebb2f
Normal-6541.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6541.png,Normal/images/Normal-6541.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6541.png,Normal/masks/Normal-6541.png,True,False,299,299,L,112.69,a9a94214644d3cebd6289028ff52ced6
Normal-6542.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6542.png,Normal/images/Normal-6542.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6542.png,Normal/masks/Normal-6542.png,True,False,299,299,L,118.59,950d24d7c9e2986103ecd87c5006de75
Normal-6543.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6543.png,Normal/images/Normal-6543.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6543.png,Normal/masks/Normal-6543.png,True,False,299,299,L,141.1,ee54101db76c0de77e0cee2bf5d2c53f
Normal-6544.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6544.png,Normal/images/Normal-6544.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6544.png,Normal/masks/Normal-6544.png,True,False,299,299,L,178.81,0b1208e4f5a77ebb20e253a35e8cb5ba
Normal-6545.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6545.png,Normal/images/Normal-6545.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6545.png,Normal/masks/Normal-6545.png,True,False,299,299,L,132.04,952071aec985e5ee172b82338040aa35
Normal-6546.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6546.png,Normal/images/Normal-6546.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6546.png,Normal/masks/Normal-6546.png,True,False,299,299,L,127.51,0f53c55b77b0f2a7285103d5b375ec27
Normal-6547.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6547.png,Normal/images/Normal-6547.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6547.png,Normal/masks/Normal-6547.png,True,False,299,299,L,172.4,6e7d42df3b43c5b0ae9b1636bfe8197c
Normal-6548.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6548.png,Normal/images/Normal-6548.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6548.png,Normal/masks/Normal-6548.png,True,False,299,299,L,116.78,6e960002685a0d6b5c3f01de63d77827
Normal-6549.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6549.png,Normal/images/Normal-6549.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6549.png,Normal/masks/Normal-6549.png,True,False,299,299,L,160.08,5ddd51da3ec66afdc0a27b81082507ce
Normal-655.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-655.png,Normal/images/Normal-655.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-655.png,Normal/masks/Normal-655.png,True,False,299,299,L,119.44,d0812bbe5f49c0138ce17a7bf553b6a5
Normal-6550.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6550.png,Normal/images/Normal-6550.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6550.png,Normal/masks/Normal-6550.png,True,False,299,299,L,132.55,22c22dc9674cc1b821c5de1aa6ac8964
Normal-6551.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6551.png,Normal/images/Normal-6551.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6551.png,Normal/masks/Normal-6551.png,True,False,299,299,L,126.18,ae038c148fdd3092b57285e65b936991
Normal-6552.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6552.png,Normal/images/Normal-6552.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6552.png,Normal/masks/Normal-6552.png,True,False,299,299,L,103.78,653186497ab59c2bfe5386353d44d791
Normal-6553.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6553.png,Normal/images/Normal-6553.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6553.png,Normal/masks/Normal-6553.png,True,False,299,299,L,138.68,49a4af2f9061c898051a9a1c99dc3205
Normal-6554.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6554.png,Normal/images/Normal-6554.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6554.png,Normal/masks/Normal-6554.png,True,False,299,299,L,161.7,d5de339c4f069c5753aa6a23be57f956
Normal-6555.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6555.png,Normal/images/Normal-6555.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6555.png,Normal/masks/Normal-6555.png,True,False,299,299,L,118.8,14dacc06493121723bab23ff14cf5c4b
Normal-6556.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6556.png,Normal/images/Normal-6556.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6556.png,Normal/masks/Normal-6556.png,True,False,299,299,L,119.32,9d9ccc098b47a77910e2e91ea34857d7
Normal-6557.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6557.png,Normal/images/Normal-6557.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6557.png,Normal/masks/Normal-6557.png,True,False,299,299,L,119.59,940dd7da70fe43b9cd7796223a3d44a9
Normal-6558.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6558.png,Normal/images/Normal-6558.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6558.png,Normal/masks/Normal-6558.png,True,False,299,299,L,161.81,9aa7b46ba2892cc31a594108ddafbb68
Normal-6559.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6559.png,Normal/images/Normal-6559.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6559.png,Normal/masks/Normal-6559.png,True,False,299,299,L,114.55,3b30500786ba30dbb47bf5639f8c6ac7
Normal-656.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-656.png,Normal/images/Normal-656.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-656.png,Normal/masks/Normal-656.png,True,False,299,299,L,121.19,e3979bf60436024c581ad1363cfe012c
Normal-6560.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6560.png,Normal/images/Normal-6560.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6560.png,Normal/masks/Normal-6560.png,True,False,299,299,L,118.45,ba4de4dec819e99abb63b252231c98f8
Normal-6561.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6561.png,Normal/images/Normal-6561.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6561.png,Normal/masks/Normal-6561.png,True,False,299,299,L,113.28,fdf99024c6a68caf028c7fc85076e152
Normal-6562.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6562.png,Normal/images/Normal-6562.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6562.png,Normal/masks/Normal-6562.png,True,False,299,299,L,116.56,8630b76a06a80924543246f5fa4a081a
Normal-6563.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6563.png,Normal/images/Normal-6563.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6563.png,Normal/masks/Normal-6563.png,True,False,299,299,L,158.78,5d2d98c56b90f42c4caf08cb2f675f83
Normal-6564.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6564.png,Normal/images/Normal-6564.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6564.png,Normal/masks/Normal-6564.png,True,False,299,299,L,169.88,048b8fd2fa737705861816b2269eeefd
Normal-6565.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6565.png,Normal/images/Normal-6565.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6565.png,Normal/masks/Normal-6565.png,True,False,299,299,L,104.77,c70a9a1fe6c51464c50c2e37eee3b2ae
Normal-6566.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6566.png,Normal/images/Normal-6566.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6566.png,Normal/masks/Normal-6566.png,True,False,299,299,L,97.42,75c972ff60ca3b6bf9023085f76fd2de
Normal-6567.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6567.png,Normal/images/Normal-6567.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6567.png,Normal/masks/Normal-6567.png,True,False,299,299,L,113.98,f68eddc234034ba022fbbc5ec6bba4e1
Normal-6568.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6568.png,Normal/images/Normal-6568.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6568.png,Normal/masks/Normal-6568.png,True,False,299,299,L,151.39,03119808afc309e935f3631ae68ab9de
Normal-6569.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6569.png,Normal/images/Normal-6569.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6569.png,Normal/masks/Normal-6569.png,True,False,299,299,L,129.03,113a83916a1eea6fe8f2c1d85f2d75cc
Normal-657.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-657.png,Normal/images/Normal-657.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-657.png,Normal/masks/Normal-657.png,True,False,299,299,L,123.17,8721bdf9a42376f97b5ed5b6908188e8
Normal-6570.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6570.png,Normal/images/Normal-6570.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6570.png,Normal/masks/Normal-6570.png,True,False,299,299,L,98.58,f04fcb290aa430c8ed3e0e02615079a6
Normal-6571.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6571.png,Normal/images/Normal-6571.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6571.png,Normal/masks/Normal-6571.png,True,False,299,299,L,178.85,0cbcf5fef7711f99876be935593e3525
Normal-6572.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6572.png,Normal/images/Normal-6572.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6572.png,Normal/masks/Normal-6572.png,True,False,299,299,L,141.02,0473da249f92484b92da4ae1867d8fe3
Normal-6573.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6573.png,Normal/images/Normal-6573.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6573.png,Normal/masks/Normal-6573.png,True,False,299,299,L,152.8,7eb40f50e71f8db43b41fb89a2f6f3e1
Normal-6574.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6574.png,Normal/images/Normal-6574.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6574.png,Normal/masks/Normal-6574.png,True,False,299,299,L,146.53,42da51366a5522f217b97a3a56bf2b73
Normal-6575.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6575.png,Normal/images/Normal-6575.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6575.png,Normal/masks/Normal-6575.png,True,False,299,299,L,134.97,f3bf1645c7f2cb4186021c6372066bfa
Normal-6576.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6576.png,Normal/images/Normal-6576.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6576.png,Normal/masks/Normal-6576.png,True,False,299,299,L,141.15,33693350cdb35ffcbcdd1f39ec5e8ce4
Normal-6577.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6577.png,Normal/images/Normal-6577.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6577.png,Normal/masks/Normal-6577.png,True,False,299,299,L,117.54,f99cb5151731ee969d437c0d5a29420d
Normal-6578.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6578.png,Normal/images/Normal-6578.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6578.png,Normal/masks/Normal-6578.png,True,False,299,299,L,172.66,4e56df4f9897aae2aa805fdd39a7a9eb
Normal-6579.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6579.png,Normal/images/Normal-6579.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6579.png,Normal/masks/Normal-6579.png,True,False,299,299,L,123.06,e77ac46968feb71e3a93ca6bc600931f
Normal-658.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-658.png,Normal/images/Normal-658.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-658.png,Normal/masks/Normal-658.png,True,False,299,299,L,85.34,4936e54b64f86232d08a1f26e76cef65
Normal-6580.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6580.png,Normal/images/Normal-6580.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6580.png,Normal/masks/Normal-6580.png,True,False,299,299,L,157.37,8fa8bcc05ce9b8fb19edfea1ce5efed6
Normal-6581.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6581.png,Normal/images/Normal-6581.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6581.png,Normal/masks/Normal-6581.png,True,False,299,299,L,122.47,0089c3aed61bb3fe95c455d869cb04ab
Normal-6582.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6582.png,Normal/images/Normal-6582.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6582.png,Normal/masks/Normal-6582.png,True,False,299,299,L,122.71,9b9ec84d50a8c3fec043b4bc70569335
Normal-6583.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6583.png,Normal/images/Normal-6583.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6583.png,Normal/masks/Normal-6583.png,True,False,299,299,L,129.09,aa1c37c1f20f0aebeacb60b5bdccfb0c
Normal-6584.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6584.png,Normal/images/Normal-6584.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6584.png,Normal/masks/Normal-6584.png,True,False,299,299,L,117.72,a48b688bfead7b3d50e1a21e0722e869
Normal-6585.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6585.png,Normal/images/Normal-6585.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6585.png,Normal/masks/Normal-6585.png,True,False,299,299,L,127.29,dcb3938fd2017ce58772bf748315277b
Normal-6586.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6586.png,Normal/images/Normal-6586.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6586.png,Normal/masks/Normal-6586.png,True,False,299,299,L,176.63,6508e1f77dc672b01629a61fd83cebeb
Normal-6587.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6587.png,Normal/images/Normal-6587.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6587.png,Normal/masks/Normal-6587.png,True,False,299,299,L,101.87,703cc8934cb5201c983591b1a2295fbe
Normal-6588.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6588.png,Normal/images/Normal-6588.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6588.png,Normal/masks/Normal-6588.png,True,False,299,299,L,123.28,6d38ac651e5941d50ec99e093290bfee
Normal-6589.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6589.png,Normal/images/Normal-6589.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6589.png,Normal/masks/Normal-6589.png,True,False,299,299,L,139.82,c2e09e043f04e85e46912b1bfccb77c8
Normal-659.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-659.png,Normal/images/Normal-659.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-659.png,Normal/masks/Normal-659.png,True,False,299,299,L,135.22,4f259a995cb965291cb7802085908a11
Normal-6590.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6590.png,Normal/images/Normal-6590.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6590.png,Normal/masks/Normal-6590.png,True,False,299,299,L,120.06,03c24a492f83ccfa34e80b382ce01741
Normal-6591.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6591.png,Normal/images/Normal-6591.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6591.png,Normal/masks/Normal-6591.png,True,False,299,299,L,167.48,5ec473e5af16d0c4d66aac24da46ac6b
Normal-6592.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6592.png,Normal/images/Normal-6592.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6592.png,Normal/masks/Normal-6592.png,True,False,299,299,L,169.07,18198c238424e19da8c9224e0a1a6513
Normal-6593.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6593.png,Normal/images/Normal-6593.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6593.png,Normal/masks/Normal-6593.png,True,False,299,299,L,124.82,58f3111ea103fa7511a8eda567df4b1b
Normal-6594.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6594.png,Normal/images/Normal-6594.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6594.png,Normal/masks/Normal-6594.png,True,False,299,299,L,113.85,b137000046efcedac2ee5846622e599b
Normal-6595.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6595.png,Normal/images/Normal-6595.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6595.png,Normal/masks/Normal-6595.png,True,False,299,299,L,154.88,3441790fbf7967db08d0d0a6b428728d
Normal-6596.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6596.png,Normal/images/Normal-6596.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6596.png,Normal/masks/Normal-6596.png,True,False,299,299,L,162.84,41ee5a9b76b21a01b80a3c186342d936
Normal-6597.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6597.png,Normal/images/Normal-6597.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6597.png,Normal/masks/Normal-6597.png,True,False,299,299,L,119.86,a1de4b3e81f0caddc3d351cc76690b03
Normal-6598.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6598.png,Normal/images/Normal-6598.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6598.png,Normal/masks/Normal-6598.png,True,False,299,299,L,115.95,87658d511cdec57cc36dd2c8fea1f14b
Normal-6599.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6599.png,Normal/images/Normal-6599.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6599.png,Normal/masks/Normal-6599.png,True,False,299,299,L,126.08,5b6239caa34602c855841f4da4ba27b4
Normal-66.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-66.png,Normal/images/Normal-66.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-66.png,Normal/masks/Normal-66.png,True,False,299,299,L,132.99,b6c34522886a4acc0466a5bee8a33697
Normal-660.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-660.png,Normal/images/Normal-660.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-660.png,Normal/masks/Normal-660.png,True,False,299,299,L,101.55,3bb72f834a0a2d16aa63fdc0bd7b2d4d
Normal-6600.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6600.png,Normal/images/Normal-6600.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6600.png,Normal/masks/Normal-6600.png,True,False,299,299,L,101.21,65d3aafba35620aae231a340aa32057a
Normal-6601.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6601.png,Normal/images/Normal-6601.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6601.png,Normal/masks/Normal-6601.png,True,False,299,299,L,107.93,119684643e7997961d1c49d3002ff872
Normal-6602.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6602.png,Normal/images/Normal-6602.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6602.png,Normal/masks/Normal-6602.png,True,False,299,299,L,163.22,93b14e63a128bb5ab2952263e6599ac7
Normal-6603.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6603.png,Normal/images/Normal-6603.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6603.png,Normal/masks/Normal-6603.png,True,False,299,299,L,122.85,d50b05814146c7871ca69f27f3b034e5
Normal-6604.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6604.png,Normal/images/Normal-6604.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6604.png,Normal/masks/Normal-6604.png,True,False,299,299,L,100.18,6c8667e49ed726a9cf1b79a8648b3268
Normal-6605.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6605.png,Normal/images/Normal-6605.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6605.png,Normal/masks/Normal-6605.png,True,False,299,299,L,139.29,e647e501318ca8f108c94050d868d6ff
Normal-6606.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6606.png,Normal/images/Normal-6606.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6606.png,Normal/masks/Normal-6606.png,True,False,299,299,L,92.98,0e84fd32f0a2c34f318c8a1fc5a3fa6d
Normal-6607.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6607.png,Normal/images/Normal-6607.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6607.png,Normal/masks/Normal-6607.png,True,False,299,299,L,143.8,3e9b30f1c79d6aeb5d431998b1c7f5df
Normal-6608.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6608.png,Normal/images/Normal-6608.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6608.png,Normal/masks/Normal-6608.png,True,False,299,299,L,132.57,a42ce34a2fcca13f78557eaebe72eb48
Normal-6609.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6609.png,Normal/images/Normal-6609.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6609.png,Normal/masks/Normal-6609.png,True,False,299,299,L,120.96,17fcebc63ee138613c10b12af79993b8
Normal-661.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-661.png,Normal/images/Normal-661.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-661.png,Normal/masks/Normal-661.png,True,False,299,299,L,113.04,ee66b916251e3b6aee3c6baa7a445fee
Normal-6610.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6610.png,Normal/images/Normal-6610.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6610.png,Normal/masks/Normal-6610.png,True,False,299,299,L,122.15,6abd375f049d43faf1b0dc10d0246071
Normal-6611.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6611.png,Normal/images/Normal-6611.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6611.png,Normal/masks/Normal-6611.png,True,False,299,299,L,107.94,8696641ad0d2b03294150b4a63652dfb
Normal-6612.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6612.png,Normal/images/Normal-6612.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6612.png,Normal/masks/Normal-6612.png,True,False,299,299,L,123.61,0cc93f00ce05218d3ad3f6bbeff6e6ef
Normal-6613.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6613.png,Normal/images/Normal-6613.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6613.png,Normal/masks/Normal-6613.png,True,False,299,299,L,106.54,6531933f7606bd09df84636ec779c2f4
Normal-6614.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6614.png,Normal/images/Normal-6614.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6614.png,Normal/masks/Normal-6614.png,True,False,299,299,L,127.47,59c38fc7371b9cf0b7b4f04e8404e09e
Normal-6615.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6615.png,Normal/images/Normal-6615.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6615.png,Normal/masks/Normal-6615.png,True,False,299,299,L,125.02,a12f71bf4a1402de64881d92bef11af4
Normal-6616.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6616.png,Normal/images/Normal-6616.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6616.png,Normal/masks/Normal-6616.png,True,False,299,299,L,71.96,160d651fe2c3ecf4a6acb9e3b1e8d942
Normal-6617.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6617.png,Normal/images/Normal-6617.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6617.png,Normal/masks/Normal-6617.png,True,False,299,299,L,119.29,90d670882a20f2a40521111856dff36d
Normal-6618.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6618.png,Normal/images/Normal-6618.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6618.png,Normal/masks/Normal-6618.png,True,False,299,299,L,119.28,df1a0c395b5e215a6c45b63cfefd0bd6
Normal-6619.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6619.png,Normal/images/Normal-6619.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6619.png,Normal/masks/Normal-6619.png,True,False,299,299,L,148.99,d89a472bafd0f3e3363487358d7e33d9
Normal-662.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-662.png,Normal/images/Normal-662.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-662.png,Normal/masks/Normal-662.png,True,False,299,299,L,125.72,7b4b48091c803f6cace7cde068d55d24
Normal-6620.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6620.png,Normal/images/Normal-6620.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6620.png,Normal/masks/Normal-6620.png,True,False,299,299,L,159.6,3e5036671d22dd46281fa7a8e1025f44
Normal-6621.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6621.png,Normal/images/Normal-6621.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6621.png,Normal/masks/Normal-6621.png,True,False,299,299,L,147.09,359ed22c8ea1c66b4190733646cde64e
Normal-6622.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6622.png,Normal/images/Normal-6622.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6622.png,Normal/masks/Normal-6622.png,True,False,299,299,L,114.96,fdf2c422596534112f20ee31541efd88
Normal-6623.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6623.png,Normal/images/Normal-6623.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6623.png,Normal/masks/Normal-6623.png,True,False,299,299,L,119.29,32b49b804d234fbcc12aa87d5e49dc31
Normal-6624.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6624.png,Normal/images/Normal-6624.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6624.png,Normal/masks/Normal-6624.png,True,False,299,299,L,156.74,7c33972b56669c7a8e1e68851733ced0
Normal-6625.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6625.png,Normal/images/Normal-6625.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6625.png,Normal/masks/Normal-6625.png,True,False,299,299,L,150.15,c233e3a2f046d49e0939ef4b0dbf95b9
Normal-6626.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6626.png,Normal/images/Normal-6626.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6626.png,Normal/masks/Normal-6626.png,True,False,299,299,L,156.73,6268c8a781db12af3eb8a75e97d6f48b
Normal-6627.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6627.png,Normal/images/Normal-6627.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6627.png,Normal/masks/Normal-6627.png,True,False,299,299,L,128.78,81adce22a6bb90ae74e18b166949b8b2
Normal-6628.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6628.png,Normal/images/Normal-6628.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6628.png,Normal/masks/Normal-6628.png,True,False,299,299,L,115.83,7cd6839c4b9e63edf66c01e863797385
Normal-6629.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6629.png,Normal/images/Normal-6629.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6629.png,Normal/masks/Normal-6629.png,True,False,299,299,L,113.68,2325602ae35b56668c206205c0b5840a
Normal-663.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-663.png,Normal/images/Normal-663.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-663.png,Normal/masks/Normal-663.png,True,False,299,299,L,134.65,dfde41d7c2d1bdd319c4509c9c9cc393
Normal-6630.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6630.png,Normal/images/Normal-6630.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6630.png,Normal/masks/Normal-6630.png,True,False,299,299,L,121.93,e36e9e8f11a2a1d262ea03a7d55a427b
Normal-6631.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6631.png,Normal/images/Normal-6631.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6631.png,Normal/masks/Normal-6631.png,True,False,299,299,L,159.55,aeb44b231685bff8b9516806c3fefd43
Normal-6632.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6632.png,Normal/images/Normal-6632.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6632.png,Normal/masks/Normal-6632.png,True,False,299,299,L,121.75,51c608dfd56c51da69f9d9a3af841f6f
Normal-6633.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6633.png,Normal/images/Normal-6633.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6633.png,Normal/masks/Normal-6633.png,True,False,299,299,L,129.31,41d6d3bdc501ad71d466ed8f40ae3035
Normal-6634.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6634.png,Normal/images/Normal-6634.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6634.png,Normal/masks/Normal-6634.png,True,False,299,299,L,112.02,a8e43e86ea6ed89ed8631e46666953c9
Normal-6635.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6635.png,Normal/images/Normal-6635.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6635.png,Normal/masks/Normal-6635.png,True,False,299,299,L,123.34,3af8ddebaec158e26d70be3ad67d6d52
Normal-6636.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6636.png,Normal/images/Normal-6636.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6636.png,Normal/masks/Normal-6636.png,True,False,299,299,L,130.4,7f65552d8951c1450edd3a9290927e8c
Normal-6637.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6637.png,Normal/images/Normal-6637.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6637.png,Normal/masks/Normal-6637.png,True,False,299,299,L,94.96,38f9045215e874bfdb26fade45021e3f
Normal-6638.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6638.png,Normal/images/Normal-6638.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6638.png,Normal/masks/Normal-6638.png,True,False,299,299,L,172.68,af2e5c67670800fa61d233ca0558a699
Normal-6639.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6639.png,Normal/images/Normal-6639.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6639.png,Normal/masks/Normal-6639.png,True,False,299,299,L,100.2,2018059f5e1ae4d28251822c31a88b6e
Normal-664.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-664.png,Normal/images/Normal-664.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-664.png,Normal/masks/Normal-664.png,True,False,299,299,L,138.02,211c51cb5e32a1cd70be43104ba59aee
Normal-6640.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6640.png,Normal/images/Normal-6640.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6640.png,Normal/masks/Normal-6640.png,True,False,299,299,L,145.89,562a9952796cad35c3a66026514f3221
Normal-6641.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6641.png,Normal/images/Normal-6641.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6641.png,Normal/masks/Normal-6641.png,True,False,299,299,L,165.1,748690a4f448c4287e66e1582c2d8b98
Normal-6642.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6642.png,Normal/images/Normal-6642.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6642.png,Normal/masks/Normal-6642.png,True,False,299,299,L,121.51,79458290587e5ceb48a7419154ac1f0b
Normal-6643.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6643.png,Normal/images/Normal-6643.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6643.png,Normal/masks/Normal-6643.png,True,False,299,299,L,107.67,c602cbd11bd4b0cd55b582eb20e6e19c
Normal-6644.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6644.png,Normal/images/Normal-6644.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6644.png,Normal/masks/Normal-6644.png,True,False,299,299,L,138.12,c525ef3a678fffff963a6083038d0a9e
Normal-6645.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6645.png,Normal/images/Normal-6645.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6645.png,Normal/masks/Normal-6645.png,True,False,299,299,L,98.22,f26762085fa932820f161e3377b2ad16
Normal-6646.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6646.png,Normal/images/Normal-6646.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6646.png,Normal/masks/Normal-6646.png,True,False,299,299,L,118.05,274bd43aaefbb5da918559be2468ca38
Normal-6647.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6647.png,Normal/images/Normal-6647.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6647.png,Normal/masks/Normal-6647.png,True,False,299,299,L,133.74,64afb7c6ac3195a51bb467922a7176ef
Normal-6648.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6648.png,Normal/images/Normal-6648.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6648.png,Normal/masks/Normal-6648.png,True,False,299,299,L,130.64,9173cee29ed77fe12d7f5078af19295b
Normal-6649.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6649.png,Normal/images/Normal-6649.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6649.png,Normal/masks/Normal-6649.png,True,False,299,299,L,105.83,85549786b961b08e5bf5a26beb7965e6
Normal-665.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-665.png,Normal/images/Normal-665.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-665.png,Normal/masks/Normal-665.png,True,False,299,299,L,124.24,a70888e500f10ad46d14958920a76f2c
Normal-6650.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6650.png,Normal/images/Normal-6650.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6650.png,Normal/masks/Normal-6650.png,True,False,299,299,L,128.7,9c34bf6f0ad0973f9d336cf2d9bd3a2a
Normal-6651.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6651.png,Normal/images/Normal-6651.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6651.png,Normal/masks/Normal-6651.png,True,False,299,299,L,108.03,ec6167223619e354ce1fffc6d0644ef3
Normal-6652.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6652.png,Normal/images/Normal-6652.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6652.png,Normal/masks/Normal-6652.png,True,False,299,299,L,165.44,9cbba919cfdb13589a38bcf1dddaabac
Normal-6653.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6653.png,Normal/images/Normal-6653.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6653.png,Normal/masks/Normal-6653.png,True,False,299,299,L,141.68,2ff683746c85987a3ff5d0cbdcfe8258
Normal-6654.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6654.png,Normal/images/Normal-6654.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6654.png,Normal/masks/Normal-6654.png,True,False,299,299,L,145.93,cca66fd5b9f230f54fed11873455787c
Normal-6655.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6655.png,Normal/images/Normal-6655.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6655.png,Normal/masks/Normal-6655.png,True,False,299,299,L,122.5,dbdac0eea12db343adc964ac51943ac4
Normal-6656.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6656.png,Normal/images/Normal-6656.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6656.png,Normal/masks/Normal-6656.png,True,False,299,299,L,164.14,eb4221707c6862b347e34b2e754c49ac
Normal-6657.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6657.png,Normal/images/Normal-6657.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6657.png,Normal/masks/Normal-6657.png,True,False,299,299,L,119.03,0dfb5fff949f3ecd34f9bba2482c9398
Normal-6658.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6658.png,Normal/images/Normal-6658.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6658.png,Normal/masks/Normal-6658.png,True,False,299,299,L,102.11,1b4b8e8b0934df7328f6c2e77570f8cf
Normal-6659.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6659.png,Normal/images/Normal-6659.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6659.png,Normal/masks/Normal-6659.png,True,False,299,299,L,116.06,8a076e38753b6139abdf56828e7286bf
Normal-666.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-666.png,Normal/images/Normal-666.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-666.png,Normal/masks/Normal-666.png,True,False,299,299,L,108.53,2b083101f4b4134793497bf9ba4dd9d9
Normal-6660.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6660.png,Normal/images/Normal-6660.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6660.png,Normal/masks/Normal-6660.png,True,False,299,299,L,104.62,e812f74b4f3e8e33d4e5d578c3bbc5b1
Normal-6661.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6661.png,Normal/images/Normal-6661.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6661.png,Normal/masks/Normal-6661.png,True,False,299,299,L,120.35,4fe25cd63ffec1ad997e8d671070807d
Normal-6662.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6662.png,Normal/images/Normal-6662.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6662.png,Normal/masks/Normal-6662.png,True,False,299,299,L,122.31,9218092b3cf5261d0b709e34cb6aceeb
Normal-6663.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6663.png,Normal/images/Normal-6663.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6663.png,Normal/masks/Normal-6663.png,True,False,299,299,L,159.09,00537f4781906e7c596ee776616eeb28
Normal-6664.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6664.png,Normal/images/Normal-6664.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6664.png,Normal/masks/Normal-6664.png,True,False,299,299,L,119.03,a2b5fac6a99356a2f133ee6fa210f23e
Normal-6665.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6665.png,Normal/images/Normal-6665.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6665.png,Normal/masks/Normal-6665.png,True,False,299,299,L,119.6,15c14cd4fb1ba368ff6ba5e04f83ac70
Normal-6666.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6666.png,Normal/images/Normal-6666.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6666.png,Normal/masks/Normal-6666.png,True,False,299,299,L,162.34,04024e1a12d7d67cce6153c6d1f5bf92
Normal-6667.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6667.png,Normal/images/Normal-6667.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6667.png,Normal/masks/Normal-6667.png,True,False,299,299,L,73.52,923166a4e6f6e59323f8a027c1fa594d
Normal-6668.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6668.png,Normal/images/Normal-6668.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6668.png,Normal/masks/Normal-6668.png,True,False,299,299,L,109.99,23b28be0fe0c87349a0e8451a579bcfc
Normal-6669.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6669.png,Normal/images/Normal-6669.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6669.png,Normal/masks/Normal-6669.png,True,False,299,299,L,144.43,667012a53c76702925de2e54bb7e8d86
Normal-667.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-667.png,Normal/images/Normal-667.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-667.png,Normal/masks/Normal-667.png,True,False,299,299,L,86.74,e26a60b1d94ec927f26a30e048bcc023
Normal-6670.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6670.png,Normal/images/Normal-6670.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6670.png,Normal/masks/Normal-6670.png,True,False,299,299,L,167.57,18a8c9a351f51f0916c73c7d3751974d
Normal-6671.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6671.png,Normal/images/Normal-6671.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6671.png,Normal/masks/Normal-6671.png,True,False,299,299,L,165.92,f5a2cafec0cd980cab2d4cbaf056a58b
Normal-6672.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6672.png,Normal/images/Normal-6672.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6672.png,Normal/masks/Normal-6672.png,True,False,299,299,L,93.68,a30a9fe17c21bb7dbe94f7101b5ba46c
Normal-6673.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6673.png,Normal/images/Normal-6673.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6673.png,Normal/masks/Normal-6673.png,True,False,299,299,L,113.25,19bfce897480bd26bc2eb429c6d3eb3b
Normal-6674.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6674.png,Normal/images/Normal-6674.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6674.png,Normal/masks/Normal-6674.png,True,False,299,299,L,113.21,f8d6892047082afceb047b9025a3e053
Normal-6675.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6675.png,Normal/images/Normal-6675.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6675.png,Normal/masks/Normal-6675.png,True,False,299,299,L,100.53,861f0aec1ac62190358e25bfe364ab46
Normal-6676.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6676.png,Normal/images/Normal-6676.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6676.png,Normal/masks/Normal-6676.png,True,False,299,299,L,131.11,546cd3cf0ebe6b5d3b43f550f9eb2120
Normal-6677.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6677.png,Normal/images/Normal-6677.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6677.png,Normal/masks/Normal-6677.png,True,False,299,299,L,123.11,40283a8bc08196e03725e7ef2a5c82e5
Normal-6678.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6678.png,Normal/images/Normal-6678.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6678.png,Normal/masks/Normal-6678.png,True,False,299,299,L,135.81,e0b84800cb97c1c22afb486d168df60b
Normal-6679.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6679.png,Normal/images/Normal-6679.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6679.png,Normal/masks/Normal-6679.png,True,False,299,299,L,113.27,c3705e3cac83cc21c3d952eeb3a9ce34
Normal-668.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-668.png,Normal/images/Normal-668.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-668.png,Normal/masks/Normal-668.png,True,False,299,299,L,120.26,4894058f3ddaa5c9e07a39bf7ab1b065
Normal-6680.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6680.png,Normal/images/Normal-6680.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6680.png,Normal/masks/Normal-6680.png,True,False,299,299,L,146.65,b5e9f63be35eaa4e708dc645fc2677b7
Normal-6681.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6681.png,Normal/images/Normal-6681.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6681.png,Normal/masks/Normal-6681.png,True,False,299,299,L,121.47,b297c8c9a76bf4e179337c60b637577d
Normal-6682.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6682.png,Normal/images/Normal-6682.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6682.png,Normal/masks/Normal-6682.png,True,False,299,299,L,106.9,55435938ad4c1382c3245351eebd9cef
Normal-6683.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6683.png,Normal/images/Normal-6683.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6683.png,Normal/masks/Normal-6683.png,True,False,299,299,L,149.49,4ce0254b282b02ca06035455ab06691e
Normal-6684.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6684.png,Normal/images/Normal-6684.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6684.png,Normal/masks/Normal-6684.png,True,False,299,299,L,127.13,67e65c3b7b8001f96a2696567507e665
Normal-6685.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6685.png,Normal/images/Normal-6685.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6685.png,Normal/masks/Normal-6685.png,True,False,299,299,L,84.93,b59a8203682de667bb4be89b1dda7a28
Normal-6686.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6686.png,Normal/images/Normal-6686.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6686.png,Normal/masks/Normal-6686.png,True,False,299,299,L,180.12,c9355242f79a314e3e42d0907c00799b
Normal-6687.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6687.png,Normal/images/Normal-6687.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6687.png,Normal/masks/Normal-6687.png,True,False,299,299,L,107.33,d8e071229e27491435a593aedac174fa
Normal-6688.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6688.png,Normal/images/Normal-6688.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6688.png,Normal/masks/Normal-6688.png,True,False,299,299,L,89.5,aff0b17f34246484abe5ca626267c320
Normal-6689.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6689.png,Normal/images/Normal-6689.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6689.png,Normal/masks/Normal-6689.png,True,False,299,299,L,167.19,3522285b6651a70c7e2e199efe63ac4c
Normal-669.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-669.png,Normal/images/Normal-669.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-669.png,Normal/masks/Normal-669.png,True,False,299,299,L,113.39,258f48527fd86926ea924370a9d78149
Normal-6690.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6690.png,Normal/images/Normal-6690.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6690.png,Normal/masks/Normal-6690.png,True,False,299,299,L,98.41,918c9c91a5490acffa308708c083df4a
Normal-6691.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6691.png,Normal/images/Normal-6691.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6691.png,Normal/masks/Normal-6691.png,True,False,299,299,L,120.53,ca47bdaeeb8198444a146cb4f058dbfe
Normal-6692.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6692.png,Normal/images/Normal-6692.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6692.png,Normal/masks/Normal-6692.png,True,False,299,299,L,140.2,ee01c7ef6b1ae98ebf296f6839407118
Normal-6693.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6693.png,Normal/images/Normal-6693.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6693.png,Normal/masks/Normal-6693.png,True,False,299,299,L,175.5,ee7e93f3679b3b3d6de5f75015b590a0
Normal-6694.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6694.png,Normal/images/Normal-6694.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6694.png,Normal/masks/Normal-6694.png,True,False,299,299,L,107.02,25a49a1409c5b1522741a095476111eb
Normal-6695.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6695.png,Normal/images/Normal-6695.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6695.png,Normal/masks/Normal-6695.png,True,False,299,299,L,92.38,67bacf5d2ac9f07dc10573d25fee2350
Normal-6696.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6696.png,Normal/images/Normal-6696.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6696.png,Normal/masks/Normal-6696.png,True,False,299,299,L,92.64,b4b6f44a3933e7bfcfc46eab17a0c6c0
Normal-6697.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6697.png,Normal/images/Normal-6697.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6697.png,Normal/masks/Normal-6697.png,True,False,299,299,L,166.76,8fd6daf848432cbaa59cac62be38403e
Normal-6698.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6698.png,Normal/images/Normal-6698.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6698.png,Normal/masks/Normal-6698.png,True,False,299,299,L,110.66,e3f3cec611a63e87e00df435484075e0
Normal-6699.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6699.png,Normal/images/Normal-6699.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6699.png,Normal/masks/Normal-6699.png,True,False,299,299,L,166.44,8961c8fb41ff9dcc2076ec07dee49b91
Normal-67.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-67.png,Normal/images/Normal-67.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-67.png,Normal/masks/Normal-67.png,True,False,299,299,L,106.23,5aaf648a6edabac71aaa7e38fb737ec4
Normal-670.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-670.png,Normal/images/Normal-670.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-670.png,Normal/masks/Normal-670.png,True,False,299,299,L,132.41,d620b5213e7530a42017018156456414
Normal-6700.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6700.png,Normal/images/Normal-6700.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6700.png,Normal/masks/Normal-6700.png,True,False,299,299,L,101.91,c08a0e17934ac58215448670ee56318f
Normal-6701.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6701.png,Normal/images/Normal-6701.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6701.png,Normal/masks/Normal-6701.png,True,False,299,299,L,113.3,f3ec9332e1940d6ffe46565f6ece2764
Normal-6702.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6702.png,Normal/images/Normal-6702.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6702.png,Normal/masks/Normal-6702.png,True,False,299,299,L,153.39,9c45687a0b8d513f37c5c917d2bb8fe3
Normal-6703.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6703.png,Normal/images/Normal-6703.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6703.png,Normal/masks/Normal-6703.png,True,False,299,299,L,155.72,080779eb135e4fd0a3255722ec5b1766
Normal-6704.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6704.png,Normal/images/Normal-6704.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6704.png,Normal/masks/Normal-6704.png,True,False,299,299,L,142.79,ffcee029c56d2a58a59dd9ecee652d77
Normal-6705.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6705.png,Normal/images/Normal-6705.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6705.png,Normal/masks/Normal-6705.png,True,False,299,299,L,99.27,3882e406151ba2c95194b29986903deb
Normal-6706.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6706.png,Normal/images/Normal-6706.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6706.png,Normal/masks/Normal-6706.png,True,False,299,299,L,111.21,34befdb452905f17baa7598925016c44
Normal-6707.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6707.png,Normal/images/Normal-6707.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6707.png,Normal/masks/Normal-6707.png,True,False,299,299,L,111.43,c25dfdd1a3ad30d4ba7c6215ff8c9590
Normal-6708.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6708.png,Normal/images/Normal-6708.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6708.png,Normal/masks/Normal-6708.png,True,False,299,299,L,137.56,f66cbade1638b64f5fd015f95187ba64
Normal-6709.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6709.png,Normal/images/Normal-6709.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6709.png,Normal/masks/Normal-6709.png,True,False,299,299,L,110.65,fb666e83f70a29f23dae5cc94ba5399b
Normal-671.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-671.png,Normal/images/Normal-671.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-671.png,Normal/masks/Normal-671.png,True,False,299,299,L,96.26,931d76e2c74a0346643a71ab6d836512
Normal-6710.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6710.png,Normal/images/Normal-6710.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6710.png,Normal/masks/Normal-6710.png,True,False,299,299,L,113.55,ec8b24d84c899c957d6fa7aa778e22d2
Normal-6711.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6711.png,Normal/images/Normal-6711.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6711.png,Normal/masks/Normal-6711.png,True,False,299,299,L,115.01,f9fdcf9337b8ecee753cf0136da77fb6
Normal-6712.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6712.png,Normal/images/Normal-6712.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6712.png,Normal/masks/Normal-6712.png,True,False,299,299,L,101.27,9e226283c130244f58eff15ed63565ba
Normal-6713.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6713.png,Normal/images/Normal-6713.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6713.png,Normal/masks/Normal-6713.png,True,False,299,299,L,165.58,a8554ada3c617f0480198426193ab7ce
Normal-6714.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6714.png,Normal/images/Normal-6714.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6714.png,Normal/masks/Normal-6714.png,True,False,299,299,L,166.35,d93343f1ed57876dcc761e710227f398
Normal-6715.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6715.png,Normal/images/Normal-6715.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6715.png,Normal/masks/Normal-6715.png,True,False,299,299,L,122.67,1ae18573798d56724834cf9638ddf304
Normal-6716.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6716.png,Normal/images/Normal-6716.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6716.png,Normal/masks/Normal-6716.png,True,False,299,299,L,116.58,9c7207a76c2e65fb62f914a9ed4b7250
Normal-6717.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6717.png,Normal/images/Normal-6717.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6717.png,Normal/masks/Normal-6717.png,True,False,299,299,L,145.03,f8c6a3bfe55b6b146c521e40bef64cae
Normal-6718.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6718.png,Normal/images/Normal-6718.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6718.png,Normal/masks/Normal-6718.png,True,False,299,299,L,127.57,b905187a3d67e6ba73b30ed73e32eed0
Normal-6719.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6719.png,Normal/images/Normal-6719.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6719.png,Normal/masks/Normal-6719.png,True,False,299,299,L,121.29,7f50f80042fbc826545a1e42277b0726
Normal-672.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-672.png,Normal/images/Normal-672.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-672.png,Normal/masks/Normal-672.png,True,False,299,299,L,118.65,ce689600ad24e37819cf85b5e923014e
Normal-6720.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6720.png,Normal/images/Normal-6720.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6720.png,Normal/masks/Normal-6720.png,True,False,299,299,L,114.27,1e57e558977d07bb372eba8945bcafaf
Normal-6721.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6721.png,Normal/images/Normal-6721.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6721.png,Normal/masks/Normal-6721.png,True,False,299,299,L,115.23,68f30a940701f25ff6f08878c09845dc
Normal-6722.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6722.png,Normal/images/Normal-6722.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6722.png,Normal/masks/Normal-6722.png,True,False,299,299,L,116.52,2568d3dd193591478b55f427ed8230ea
Normal-6723.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6723.png,Normal/images/Normal-6723.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6723.png,Normal/masks/Normal-6723.png,True,False,299,299,L,124.07,c9da9c46ec60651085ac916d5c6eea4d
Normal-6724.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6724.png,Normal/images/Normal-6724.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6724.png,Normal/masks/Normal-6724.png,True,False,299,299,L,143.31,abc27d0ca6b2d3fefde7b2372d0c5a88
Normal-6725.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6725.png,Normal/images/Normal-6725.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6725.png,Normal/masks/Normal-6725.png,True,False,299,299,L,106.85,a94d6602021f41e0121434bb925a6135
Normal-6726.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6726.png,Normal/images/Normal-6726.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6726.png,Normal/masks/Normal-6726.png,True,False,299,299,L,98.67,8fc4e7d8883abebd9e567271969473c5
Normal-6727.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6727.png,Normal/images/Normal-6727.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6727.png,Normal/masks/Normal-6727.png,True,False,299,299,L,165.72,610049017798d60263bd58bbe620fc66
Normal-6728.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6728.png,Normal/images/Normal-6728.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6728.png,Normal/masks/Normal-6728.png,True,False,299,299,L,141.29,4aa6568ec469e66497d9254b5651eabc
Normal-6729.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6729.png,Normal/images/Normal-6729.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6729.png,Normal/masks/Normal-6729.png,True,False,299,299,L,125.02,baf16ceaac85812079bb5a296150b9cd
Normal-673.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-673.png,Normal/images/Normal-673.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-673.png,Normal/masks/Normal-673.png,True,False,299,299,L,114.34,b1de634607b7e29435d90839aa67da9e
Normal-6730.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6730.png,Normal/images/Normal-6730.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6730.png,Normal/masks/Normal-6730.png,True,False,299,299,L,140.44,4031e9b1a20f7faf9d4fe6471b5a60e3
Normal-6731.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6731.png,Normal/images/Normal-6731.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6731.png,Normal/masks/Normal-6731.png,True,False,299,299,L,143.66,d4ab203e71b17b3af77a9fcbe9eb108f
Normal-6732.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6732.png,Normal/images/Normal-6732.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6732.png,Normal/masks/Normal-6732.png,True,False,299,299,L,114.77,1a3a736f4e948b36edbc1a7cc26e79fc
Normal-6733.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6733.png,Normal/images/Normal-6733.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6733.png,Normal/masks/Normal-6733.png,True,False,299,299,L,136.53,c7cf1c542f8416559a9c8e6b6390fca9
Normal-6734.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6734.png,Normal/images/Normal-6734.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6734.png,Normal/masks/Normal-6734.png,True,False,299,299,L,159.81,d4502cceaa4688fca609fd90624b4bde
Normal-6735.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6735.png,Normal/images/Normal-6735.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6735.png,Normal/masks/Normal-6735.png,True,False,299,299,L,168.85,906fda0d9b792e67a93b49fc33589be2
Normal-6736.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6736.png,Normal/images/Normal-6736.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6736.png,Normal/masks/Normal-6736.png,True,False,299,299,L,131.71,129bd3c08db7ce767eacf0ab46075e1c
Normal-6737.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6737.png,Normal/images/Normal-6737.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6737.png,Normal/masks/Normal-6737.png,True,False,299,299,L,126.15,017b8fbfcf9f673ffa454d2d1da0a0af
Normal-6738.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6738.png,Normal/images/Normal-6738.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6738.png,Normal/masks/Normal-6738.png,True,False,299,299,L,109.02,9b71eaf64df49c4140476976c2fa419a
Normal-6739.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6739.png,Normal/images/Normal-6739.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6739.png,Normal/masks/Normal-6739.png,True,False,299,299,L,113.91,8a285ca5b44e39c66a8480aa54eeb24e
Normal-674.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-674.png,Normal/images/Normal-674.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-674.png,Normal/masks/Normal-674.png,True,False,299,299,L,99.74,fc52bc6475a279cd60421cdc30853f39
Normal-6740.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6740.png,Normal/images/Normal-6740.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6740.png,Normal/masks/Normal-6740.png,True,False,299,299,L,135.46,0362a27b5e1fc9288e8ef41106792b4b
Normal-6741.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6741.png,Normal/images/Normal-6741.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6741.png,Normal/masks/Normal-6741.png,True,False,299,299,L,116.49,6021d0b5c284965e0456ff9503b958e5
Normal-6742.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6742.png,Normal/images/Normal-6742.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6742.png,Normal/masks/Normal-6742.png,True,False,299,299,L,125.85,b8d1a161b155f9aafa653aa5ab9e590c
Normal-6743.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6743.png,Normal/images/Normal-6743.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6743.png,Normal/masks/Normal-6743.png,True,False,299,299,L,137.85,05dc14075e8f0ec37c3a30a573b39244
Normal-6744.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6744.png,Normal/images/Normal-6744.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6744.png,Normal/masks/Normal-6744.png,True,False,299,299,L,99.23,d9f1daee0fdf3930c6fb0b2f5f2d3eb8
Normal-6745.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6745.png,Normal/images/Normal-6745.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6745.png,Normal/masks/Normal-6745.png,True,False,299,299,L,115.07,0766638a43988885bdd94d348dd389f0
Normal-6746.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6746.png,Normal/images/Normal-6746.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6746.png,Normal/masks/Normal-6746.png,True,False,299,299,L,118.74,5966e72576f6f0ee5a5951ddbdd93cfb
Normal-6747.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6747.png,Normal/images/Normal-6747.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6747.png,Normal/masks/Normal-6747.png,True,False,299,299,L,133.03,15086874b796528dbf0d6432d3c9e3e2
Normal-6748.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6748.png,Normal/images/Normal-6748.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6748.png,Normal/masks/Normal-6748.png,True,False,299,299,L,124.58,7badc3c9fb4c83f3dc6d347b73053ced
Normal-6749.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6749.png,Normal/images/Normal-6749.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6749.png,Normal/masks/Normal-6749.png,True,False,299,299,L,123.73,fe2dcd60e6768d2724a1ddb2f9d685b8
Normal-675.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-675.png,Normal/images/Normal-675.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-675.png,Normal/masks/Normal-675.png,True,False,299,299,L,120.16,d8e747633a134a98cf629b77f4d33133
Normal-6750.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6750.png,Normal/images/Normal-6750.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6750.png,Normal/masks/Normal-6750.png,True,False,299,299,L,141.58,eb2c738a8001a85886f11cdfe75b2269
Normal-6751.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6751.png,Normal/images/Normal-6751.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6751.png,Normal/masks/Normal-6751.png,True,False,299,299,L,120.54,03c2776bacb28484e2a59f2145255f9f
Normal-6752.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6752.png,Normal/images/Normal-6752.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6752.png,Normal/masks/Normal-6752.png,True,False,299,299,L,105.29,494749f375773eccee08aaffb32284d5
Normal-6753.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6753.png,Normal/images/Normal-6753.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6753.png,Normal/masks/Normal-6753.png,True,False,299,299,L,104.65,c0ba9dc69a2f4a666c01eef9560b249f
Normal-6754.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6754.png,Normal/images/Normal-6754.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6754.png,Normal/masks/Normal-6754.png,True,False,299,299,L,135.95,6c0f1d8eb4769073b21a3699cd673988
Normal-6755.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6755.png,Normal/images/Normal-6755.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6755.png,Normal/masks/Normal-6755.png,True,False,299,299,L,125.51,d077e5244fadb5a35aede8773a516020
Normal-6756.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6756.png,Normal/images/Normal-6756.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6756.png,Normal/masks/Normal-6756.png,True,False,299,299,L,145.73,0612dc1a5bd8050b0a10c0ffc6633bb0
Normal-6757.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6757.png,Normal/images/Normal-6757.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6757.png,Normal/masks/Normal-6757.png,True,False,299,299,L,121.51,4bcd3326439cabfd6a32126c18748075
Normal-6758.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6758.png,Normal/images/Normal-6758.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6758.png,Normal/masks/Normal-6758.png,True,False,299,299,L,176.1,a99e794a83a6987e3f4454e88bb76106
Normal-6759.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6759.png,Normal/images/Normal-6759.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6759.png,Normal/masks/Normal-6759.png,True,False,299,299,L,108.05,8a9a6bcf17b6ee974c29fd0d577c0fba
Normal-676.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-676.png,Normal/images/Normal-676.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-676.png,Normal/masks/Normal-676.png,True,False,299,299,L,135.02,84f9e92d7149f010e55a199f9f685958
Normal-6760.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6760.png,Normal/images/Normal-6760.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6760.png,Normal/masks/Normal-6760.png,True,False,299,299,L,128.71,3e906cea0031f121de3d9c71efb9fe4e
Normal-6761.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6761.png,Normal/images/Normal-6761.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6761.png,Normal/masks/Normal-6761.png,True,False,299,299,L,176.71,f54379ca9837340243c7b8f03d5cbb60
Normal-6762.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6762.png,Normal/images/Normal-6762.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6762.png,Normal/masks/Normal-6762.png,True,False,299,299,L,105.02,18c5c291d524246ae614b468d2cddc86
Normal-6763.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6763.png,Normal/images/Normal-6763.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6763.png,Normal/masks/Normal-6763.png,True,False,299,299,L,139.95,b53dde8672c5adfee4ce0ebe0e30bfb9
Normal-6764.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6764.png,Normal/images/Normal-6764.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6764.png,Normal/masks/Normal-6764.png,True,False,299,299,L,136.9,d4232a2a781beb44d5a1372210ba5d4f
Normal-6765.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6765.png,Normal/images/Normal-6765.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6765.png,Normal/masks/Normal-6765.png,True,False,299,299,L,131.8,b15109926229733b833fa950ce101e27
Normal-6766.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6766.png,Normal/images/Normal-6766.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6766.png,Normal/masks/Normal-6766.png,True,False,299,299,L,116.3,515cb35a53fac0a4bc74b3e66fb870c0
Normal-6767.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6767.png,Normal/images/Normal-6767.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6767.png,Normal/masks/Normal-6767.png,True,False,299,299,L,115.0,f7bef1587fe4febec588ba94a91ada94
Normal-6768.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6768.png,Normal/images/Normal-6768.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6768.png,Normal/masks/Normal-6768.png,True,False,299,299,L,142.03,6f47af2baedab703cf9a9fa224aff75e
Normal-6769.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6769.png,Normal/images/Normal-6769.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6769.png,Normal/masks/Normal-6769.png,True,False,299,299,L,148.29,d6def8a6796a7ea2455b07e5294b563a
Normal-677.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-677.png,Normal/images/Normal-677.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-677.png,Normal/masks/Normal-677.png,True,False,299,299,L,131.9,2515a2fc393399462a887341fa4cfad6
Normal-6770.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6770.png,Normal/images/Normal-6770.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6770.png,Normal/masks/Normal-6770.png,True,False,299,299,L,164.64,f96daa44a874ef84e801e58616b33c82
Normal-6771.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6771.png,Normal/images/Normal-6771.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6771.png,Normal/masks/Normal-6771.png,True,False,299,299,L,124.08,d8a1f1269aa6d3cd583b910c4ead39e7
Normal-6772.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6772.png,Normal/images/Normal-6772.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6772.png,Normal/masks/Normal-6772.png,True,False,299,299,L,142.71,a3d1e7c3a737339855ae76af6e86d522
Normal-6773.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6773.png,Normal/images/Normal-6773.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6773.png,Normal/masks/Normal-6773.png,True,False,299,299,L,137.16,aa0b15480c735ecab928c6c251e9a252
Normal-6774.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6774.png,Normal/images/Normal-6774.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6774.png,Normal/masks/Normal-6774.png,True,False,299,299,L,96.17,1ec6c1e837d9acf9bc64493ec0cfba84
Normal-6775.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6775.png,Normal/images/Normal-6775.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6775.png,Normal/masks/Normal-6775.png,True,False,299,299,L,109.51,57a69b1d90e189c28d9bc2dace05d8ca
Normal-6776.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6776.png,Normal/images/Normal-6776.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6776.png,Normal/masks/Normal-6776.png,True,False,299,299,L,87.72,5bb774cbee8ebd19b4bc138e3136c4d1
Normal-6777.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6777.png,Normal/images/Normal-6777.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6777.png,Normal/masks/Normal-6777.png,True,False,299,299,L,134.86,ead7adc7681195134f82bc578bf4a4c0
Normal-6778.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6778.png,Normal/images/Normal-6778.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6778.png,Normal/masks/Normal-6778.png,True,False,299,299,L,107.87,d9fd7c073f5597b9e8164436948d7f8b
Normal-6779.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6779.png,Normal/images/Normal-6779.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6779.png,Normal/masks/Normal-6779.png,True,False,299,299,L,180.24,e2315efaa6cc96cbc1f3d1f46c127cc6
Normal-678.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-678.png,Normal/images/Normal-678.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-678.png,Normal/masks/Normal-678.png,True,False,299,299,L,106.02,2819119629cf40d348582de98f3b0eb5
Normal-6780.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6780.png,Normal/images/Normal-6780.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6780.png,Normal/masks/Normal-6780.png,True,False,299,299,L,124.07,9837baca528ac02866db871c8e960c0f
Normal-6781.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6781.png,Normal/images/Normal-6781.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6781.png,Normal/masks/Normal-6781.png,True,False,299,299,L,130.61,59dc31afc432d49e94131aa2a5c45a43
Normal-6782.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6782.png,Normal/images/Normal-6782.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6782.png,Normal/masks/Normal-6782.png,True,False,299,299,L,97.68,6651b5383a4faf58484c50e3ebc4794a
Normal-6783.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6783.png,Normal/images/Normal-6783.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6783.png,Normal/masks/Normal-6783.png,True,False,299,299,L,119.08,1333a05ceec2893beffbe9c68ec1a9ac
Normal-6784.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6784.png,Normal/images/Normal-6784.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6784.png,Normal/masks/Normal-6784.png,True,False,299,299,L,133.0,774b6016cdc57cd3290603ceec4f86ad
Normal-6785.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6785.png,Normal/images/Normal-6785.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6785.png,Normal/masks/Normal-6785.png,True,False,299,299,L,98.23,0316fadbd1a2ec4444153e7b9a08e3da
Normal-6786.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6786.png,Normal/images/Normal-6786.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6786.png,Normal/masks/Normal-6786.png,True,False,299,299,L,160.48,e68241588ebd199a7e187fc69f601f85
Normal-6787.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6787.png,Normal/images/Normal-6787.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6787.png,Normal/masks/Normal-6787.png,True,False,299,299,L,117.59,ec58731717ca973cae86f5cbd681c147
Normal-6788.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6788.png,Normal/images/Normal-6788.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6788.png,Normal/masks/Normal-6788.png,True,False,299,299,L,109.47,3aef51db83c78adf713b66f6fff16f20
Normal-6789.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6789.png,Normal/images/Normal-6789.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6789.png,Normal/masks/Normal-6789.png,True,False,299,299,L,108.48,08db7d0d80aa64f007e3e331412d1f41
Normal-679.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-679.png,Normal/images/Normal-679.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-679.png,Normal/masks/Normal-679.png,True,False,299,299,L,117.06,4208d07325d64f6610dac006c5f12051
Normal-6790.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6790.png,Normal/images/Normal-6790.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6790.png,Normal/masks/Normal-6790.png,True,False,299,299,L,113.22,eeda9336e07c66407facebda66bfa88e
Normal-6791.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6791.png,Normal/images/Normal-6791.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6791.png,Normal/masks/Normal-6791.png,True,False,299,299,L,174.74,4118d42f071e254b273352322901f36a
Normal-6792.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6792.png,Normal/images/Normal-6792.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6792.png,Normal/masks/Normal-6792.png,True,False,299,299,L,134.09,c574eb179a6b8b22ad3f065c6c10b191
Normal-6793.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6793.png,Normal/images/Normal-6793.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6793.png,Normal/masks/Normal-6793.png,True,False,299,299,L,131.68,c6c7549947e38a6f7c684b9de6e5396b
Normal-6794.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6794.png,Normal/images/Normal-6794.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6794.png,Normal/masks/Normal-6794.png,True,False,299,299,L,122.5,21ad734f1be96d197f24de286d9b3d30
Normal-6795.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6795.png,Normal/images/Normal-6795.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6795.png,Normal/masks/Normal-6795.png,True,False,299,299,L,177.73,79567da4455fc91c80d4e8c4ccb1fb79
Normal-6796.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6796.png,Normal/images/Normal-6796.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6796.png,Normal/masks/Normal-6796.png,True,False,299,299,L,108.57,67cbf108570d761a77595fdb0c9e555d
Normal-6797.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6797.png,Normal/images/Normal-6797.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6797.png,Normal/masks/Normal-6797.png,True,False,299,299,L,124.92,5c501ce432dcb9bf87ceab1590d95177
Normal-6798.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6798.png,Normal/images/Normal-6798.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6798.png,Normal/masks/Normal-6798.png,True,False,299,299,L,179.89,30d0d23c55e26917390f0312e353dc9d
Normal-6799.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6799.png,Normal/images/Normal-6799.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6799.png,Normal/masks/Normal-6799.png,True,False,299,299,L,129.17,0f2eea6b73c5fe144a82ae83e36471fd
Normal-68.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-68.png,Normal/images/Normal-68.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-68.png,Normal/masks/Normal-68.png,True,False,299,299,L,130.38,f1519f7c2da31c71ad9bc0a1c49f3b60
Normal-680.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-680.png,Normal/images/Normal-680.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-680.png,Normal/masks/Normal-680.png,True,False,299,299,L,135.7,3a2aaa0b3c6021a20e3438c757807006
Normal-6800.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6800.png,Normal/images/Normal-6800.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6800.png,Normal/masks/Normal-6800.png,True,False,299,299,L,114.8,513c6e92021a4b57e427f65bcc8110ae
Normal-6801.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6801.png,Normal/images/Normal-6801.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6801.png,Normal/masks/Normal-6801.png,True,False,299,299,L,146.91,c760dc6ba2c6655e89434164991d4a86
Normal-6802.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6802.png,Normal/images/Normal-6802.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6802.png,Normal/masks/Normal-6802.png,True,False,299,299,L,111.46,19a140084b4a40d19759606b9f610fc4
Normal-6803.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6803.png,Normal/images/Normal-6803.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6803.png,Normal/masks/Normal-6803.png,True,False,299,299,L,111.33,8af33d6fc134cba9aa41b4a643cb502d
Normal-6804.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6804.png,Normal/images/Normal-6804.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6804.png,Normal/masks/Normal-6804.png,True,False,299,299,L,149.28,8f7d159392752d1e1a39cc5296f5feb5
Normal-6805.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6805.png,Normal/images/Normal-6805.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6805.png,Normal/masks/Normal-6805.png,True,False,299,299,L,163.15,652772653184a56c8c8620b3b98ab740
Normal-6806.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6806.png,Normal/images/Normal-6806.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6806.png,Normal/masks/Normal-6806.png,True,False,299,299,L,90.25,c3e885b44e1fa9d8c9f85ddab18a2a3d
Normal-6807.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6807.png,Normal/images/Normal-6807.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6807.png,Normal/masks/Normal-6807.png,True,False,299,299,L,128.62,12be5a61b9d22783453359ae5730da6a
Normal-6808.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6808.png,Normal/images/Normal-6808.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6808.png,Normal/masks/Normal-6808.png,True,False,299,299,L,110.49,95c07c75e8965411533b5ec7fbb37395
Normal-6809.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6809.png,Normal/images/Normal-6809.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6809.png,Normal/masks/Normal-6809.png,True,False,299,299,L,108.76,e9d3b2748b955737274269c3b2c0d9d0
Normal-681.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-681.png,Normal/images/Normal-681.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-681.png,Normal/masks/Normal-681.png,True,False,299,299,L,90.76,20053cfda6b34bb78ec4ffc967acbf34
Normal-6810.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6810.png,Normal/images/Normal-6810.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6810.png,Normal/masks/Normal-6810.png,True,False,299,299,L,97.96,b3e704903cb4bf0cadc28c0310b31a4b
Normal-6811.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6811.png,Normal/images/Normal-6811.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6811.png,Normal/masks/Normal-6811.png,True,False,299,299,L,148.16,b910ec70c57df06d73d90709562003d2
Normal-6812.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6812.png,Normal/images/Normal-6812.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6812.png,Normal/masks/Normal-6812.png,True,False,299,299,L,134.26,6819618b634e57e3f5d52bf5e122b6f6
Normal-6813.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6813.png,Normal/images/Normal-6813.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6813.png,Normal/masks/Normal-6813.png,True,False,299,299,L,84.15,cf771c93b069cdb63151aa1f9d1fb2c9
Normal-6814.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6814.png,Normal/images/Normal-6814.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6814.png,Normal/masks/Normal-6814.png,True,False,299,299,L,109.87,ef7846e9e7102ae36f28ea9e16386da9
Normal-6815.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6815.png,Normal/images/Normal-6815.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6815.png,Normal/masks/Normal-6815.png,True,False,299,299,L,108.51,cf421b404de1393d8720814aa78a0444
Normal-6816.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6816.png,Normal/images/Normal-6816.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6816.png,Normal/masks/Normal-6816.png,True,False,299,299,L,112.86,c9a203f2c46055d774cd885e71227bad
Normal-6817.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6817.png,Normal/images/Normal-6817.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6817.png,Normal/masks/Normal-6817.png,True,False,299,299,L,131.96,4039e338203aa3ead1c95a6d45988262
Normal-6818.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6818.png,Normal/images/Normal-6818.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6818.png,Normal/masks/Normal-6818.png,True,False,299,299,L,124.59,6ca3fc9aebeabfe142438cec59bd122a
Normal-6819.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6819.png,Normal/images/Normal-6819.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6819.png,Normal/masks/Normal-6819.png,True,False,299,299,L,111.91,335067426cdbbffaea0449ca0e7794a2
Normal-682.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-682.png,Normal/images/Normal-682.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-682.png,Normal/masks/Normal-682.png,True,False,299,299,L,106.46,5741baf9d795297b7c4af7bf9b6a2e42
Normal-6820.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6820.png,Normal/images/Normal-6820.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6820.png,Normal/masks/Normal-6820.png,True,False,299,299,L,135.78,428c06192a0686490e423fbb4d830efe
Normal-6821.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6821.png,Normal/images/Normal-6821.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6821.png,Normal/masks/Normal-6821.png,True,False,299,299,L,153.14,7974650ffbf7a618a7ff0d0b228c256e
Normal-6822.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6822.png,Normal/images/Normal-6822.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6822.png,Normal/masks/Normal-6822.png,True,False,299,299,L,172.63,95019992d406f1d9f1fd61529db6219e
Normal-6823.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6823.png,Normal/images/Normal-6823.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6823.png,Normal/masks/Normal-6823.png,True,False,299,299,L,109.6,d8a882727c0cab00e5868758da0d7516
Normal-6824.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6824.png,Normal/images/Normal-6824.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6824.png,Normal/masks/Normal-6824.png,True,False,299,299,L,132.77,f308010592438ff0f2992e739cc8e271
Normal-6825.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6825.png,Normal/images/Normal-6825.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6825.png,Normal/masks/Normal-6825.png,True,False,299,299,L,127.88,83f614d14b268d24ca7ee946897d357d
Normal-6826.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6826.png,Normal/images/Normal-6826.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6826.png,Normal/masks/Normal-6826.png,True,False,299,299,L,125.92,d2933a606da09291d3c374e5e18c582a
Normal-6827.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6827.png,Normal/images/Normal-6827.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6827.png,Normal/masks/Normal-6827.png,True,False,299,299,L,153.66,fc1ee0eb37f17858317621b7b2ee8de4
Normal-6828.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6828.png,Normal/images/Normal-6828.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6828.png,Normal/masks/Normal-6828.png,True,False,299,299,L,160.08,e04fcf55ce0c96fdbee779b191ba6f01
Normal-6829.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6829.png,Normal/images/Normal-6829.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6829.png,Normal/masks/Normal-6829.png,True,False,299,299,L,87.42,765a4492e7e8ebdffa9ba53a76a1f79b
Normal-683.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-683.png,Normal/images/Normal-683.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-683.png,Normal/masks/Normal-683.png,True,False,299,299,L,115.09,cdf6b62c23148696e10af217db670cc2
Normal-6830.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6830.png,Normal/images/Normal-6830.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6830.png,Normal/masks/Normal-6830.png,True,False,299,299,L,166.16,e581cc1adaa8189b39550b0ac149d614
Normal-6831.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6831.png,Normal/images/Normal-6831.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6831.png,Normal/masks/Normal-6831.png,True,False,299,299,L,127.6,613f7bd4815ddf54a9c459f3e14365cf
Normal-6832.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6832.png,Normal/images/Normal-6832.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6832.png,Normal/masks/Normal-6832.png,True,False,299,299,L,161.57,c8eace6da436bb1029cc882eefd236f5
Normal-6833.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6833.png,Normal/images/Normal-6833.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6833.png,Normal/masks/Normal-6833.png,True,False,299,299,L,105.94,83f7d5abe7df74ea58b5cb129ca27e1f
Normal-6834.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6834.png,Normal/images/Normal-6834.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6834.png,Normal/masks/Normal-6834.png,True,False,299,299,L,119.92,a5ae1191fb63ebdcbaf995a12fe1c5e2
Normal-6835.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6835.png,Normal/images/Normal-6835.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6835.png,Normal/masks/Normal-6835.png,True,False,299,299,L,123.47,2eb3c6f7ea3a50ac5ed2b1afceedd161
Normal-6836.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6836.png,Normal/images/Normal-6836.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6836.png,Normal/masks/Normal-6836.png,True,False,299,299,L,125.97,dbcea162bc1d74fe269127d64e311f54
Normal-6837.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6837.png,Normal/images/Normal-6837.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6837.png,Normal/masks/Normal-6837.png,True,False,299,299,L,127.82,5e9cd6fb566c6fefbdb1177579bad03d
Normal-6838.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6838.png,Normal/images/Normal-6838.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6838.png,Normal/masks/Normal-6838.png,True,False,299,299,L,171.93,1fdcc3ca383ddb8b4db9fc4ded03bd4a
Normal-6839.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6839.png,Normal/images/Normal-6839.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6839.png,Normal/masks/Normal-6839.png,True,False,299,299,L,120.2,677d83042535feb1e11e9a18648239df
Normal-684.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-684.png,Normal/images/Normal-684.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-684.png,Normal/masks/Normal-684.png,True,False,299,299,L,128.04,c56895af4b072d46cdb97ebdd4dd2219
Normal-6840.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6840.png,Normal/images/Normal-6840.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6840.png,Normal/masks/Normal-6840.png,True,False,299,299,L,146.67,a9f0ff15f3f41aa03b7a33519f42cea0
Normal-6841.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6841.png,Normal/images/Normal-6841.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6841.png,Normal/masks/Normal-6841.png,True,False,299,299,L,112.52,492938a3979ac5613a58906d7661846d
Normal-6842.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6842.png,Normal/images/Normal-6842.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6842.png,Normal/masks/Normal-6842.png,True,False,299,299,L,84.22,e254cb66f2ce659d4575855ae9ca7777
Normal-6843.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6843.png,Normal/images/Normal-6843.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6843.png,Normal/masks/Normal-6843.png,True,False,299,299,L,148.52,ab19b6715f31e76612b97d9c48ca3125
Normal-6844.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6844.png,Normal/images/Normal-6844.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6844.png,Normal/masks/Normal-6844.png,True,False,299,299,L,165.71,964b3a60d84eb2abbd5a40acf9f4b146
Normal-6845.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6845.png,Normal/images/Normal-6845.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6845.png,Normal/masks/Normal-6845.png,True,False,299,299,L,171.53,0a8ce86e9833e7471c558c9c0d9e64e5
Normal-6846.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6846.png,Normal/images/Normal-6846.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6846.png,Normal/masks/Normal-6846.png,True,False,299,299,L,106.42,38c5c2b91231d55415b2b3de279d4564
Normal-6847.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6847.png,Normal/images/Normal-6847.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6847.png,Normal/masks/Normal-6847.png,True,False,299,299,L,129.97,09388ebfb0fb2925e2e889c2792f8be2
Normal-6848.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6848.png,Normal/images/Normal-6848.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6848.png,Normal/masks/Normal-6848.png,True,False,299,299,L,160.51,10434d8d173b42ee8351c00cf09b0009
Normal-6849.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6849.png,Normal/images/Normal-6849.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6849.png,Normal/masks/Normal-6849.png,True,False,299,299,L,121.12,e3e5ebc2d78ba925466225d8262fb053
Normal-685.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-685.png,Normal/images/Normal-685.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-685.png,Normal/masks/Normal-685.png,True,False,299,299,L,130.78,9070b826bdc60e33b0017a6904d52821
Normal-6850.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6850.png,Normal/images/Normal-6850.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6850.png,Normal/masks/Normal-6850.png,True,False,299,299,L,131.64,eaa311f760ab392baf640ffdcd5b5b38
Normal-6851.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6851.png,Normal/images/Normal-6851.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6851.png,Normal/masks/Normal-6851.png,True,False,299,299,L,171.27,f047bf0b02f3e24c682e0d236bc1239f
Normal-6852.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6852.png,Normal/images/Normal-6852.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6852.png,Normal/masks/Normal-6852.png,True,False,299,299,L,164.33,895db3fc641ef739a81cc82a829d932a
Normal-6853.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6853.png,Normal/images/Normal-6853.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6853.png,Normal/masks/Normal-6853.png,True,False,299,299,L,175.98,98fae62948cc53b3f3dfc32186f05435
Normal-6854.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6854.png,Normal/images/Normal-6854.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6854.png,Normal/masks/Normal-6854.png,True,False,299,299,L,130.42,3b939126ae9a741b8d9cacf4d1496fec
Normal-6855.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6855.png,Normal/images/Normal-6855.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6855.png,Normal/masks/Normal-6855.png,True,False,299,299,L,74.61,3b88503b21dd72831c56347c4b732486
Normal-6856.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6856.png,Normal/images/Normal-6856.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6856.png,Normal/masks/Normal-6856.png,True,False,299,299,L,151.93,761683d2240092bda89d15967ccb6ae0
Normal-6857.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6857.png,Normal/images/Normal-6857.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6857.png,Normal/masks/Normal-6857.png,True,False,299,299,L,177.57,9c28904fc55bb8f97325f8ed21c229bc
Normal-6858.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6858.png,Normal/images/Normal-6858.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6858.png,Normal/masks/Normal-6858.png,True,False,299,299,L,134.31,9df7eaece2106a55f8d11ab9be3e44e2
Normal-6859.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6859.png,Normal/images/Normal-6859.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6859.png,Normal/masks/Normal-6859.png,True,False,299,299,L,115.82,91718cbb5749271b83da22deff487a18
Normal-686.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-686.png,Normal/images/Normal-686.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-686.png,Normal/masks/Normal-686.png,True,False,299,299,L,125.35,52db1e8bb5ef3274fc830edacc9c0178
Normal-6860.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6860.png,Normal/images/Normal-6860.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6860.png,Normal/masks/Normal-6860.png,True,False,299,299,L,165.86,8f66f4c8c220ef487d564eee33852bc1
Normal-6861.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6861.png,Normal/images/Normal-6861.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6861.png,Normal/masks/Normal-6861.png,True,False,299,299,L,146.81,92f0c05725c7dbb448d1f76ab5edb72b
Normal-6862.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6862.png,Normal/images/Normal-6862.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6862.png,Normal/masks/Normal-6862.png,True,False,299,299,L,174.61,64297e4e590245efac0dcc32fcd4b73e
Normal-6863.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6863.png,Normal/images/Normal-6863.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6863.png,Normal/masks/Normal-6863.png,True,False,299,299,L,119.21,92c5c7a2d54440271f4143037daed179
Normal-6864.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6864.png,Normal/images/Normal-6864.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6864.png,Normal/masks/Normal-6864.png,True,False,299,299,L,106.33,2666b7556b8fe1ded41c95fe635880ed
Normal-6865.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6865.png,Normal/images/Normal-6865.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6865.png,Normal/masks/Normal-6865.png,True,False,299,299,L,107.84,8dba7e7d1b5262f4934279b84fc8ec4d
Normal-6866.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6866.png,Normal/images/Normal-6866.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6866.png,Normal/masks/Normal-6866.png,True,False,299,299,L,128.94,47ad4384e9bb5a034ff9688cf8890164
Normal-6867.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6867.png,Normal/images/Normal-6867.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6867.png,Normal/masks/Normal-6867.png,True,False,299,299,L,138.54,fc7ede43e7d4e6dea6386e4ca982bc7b
Normal-6868.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6868.png,Normal/images/Normal-6868.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6868.png,Normal/masks/Normal-6868.png,True,False,299,299,L,167.71,8ee7ab9749755807210616f2314f8cf1
Normal-6869.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6869.png,Normal/images/Normal-6869.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6869.png,Normal/masks/Normal-6869.png,True,False,299,299,L,110.08,d2b06434d0048ecb61344ed7be614a72
Normal-687.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-687.png,Normal/images/Normal-687.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-687.png,Normal/masks/Normal-687.png,True,False,299,299,L,132.33,f1db2446b8f23b72122c0412d4752ea6
Normal-6870.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6870.png,Normal/images/Normal-6870.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6870.png,Normal/masks/Normal-6870.png,True,False,299,299,L,107.81,d0058aa959e98b8ff8aa442e8fc34791
Normal-6871.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6871.png,Normal/images/Normal-6871.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6871.png,Normal/masks/Normal-6871.png,True,False,299,299,L,165.81,5bcaea34ac6768a484f885f01c177ef9
Normal-6872.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6872.png,Normal/images/Normal-6872.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6872.png,Normal/masks/Normal-6872.png,True,False,299,299,L,128.44,5d060da9b251c72b6c215c4195c691cf
Normal-6873.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6873.png,Normal/images/Normal-6873.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6873.png,Normal/masks/Normal-6873.png,True,False,299,299,L,159.21,5ea81c0f48ba6088393b97cab32284e1
Normal-6874.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6874.png,Normal/images/Normal-6874.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6874.png,Normal/masks/Normal-6874.png,True,False,299,299,L,112.88,f88e9d8c17a9f16b57915628ce81ef24
Normal-6875.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6875.png,Normal/images/Normal-6875.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6875.png,Normal/masks/Normal-6875.png,True,False,299,299,L,127.39,888955a10997965a49c8fd24ed51a5a6
Normal-6876.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6876.png,Normal/images/Normal-6876.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6876.png,Normal/masks/Normal-6876.png,True,False,299,299,L,154.68,a80f9c94cd62db0059a83926eeb59b7a
Normal-6877.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6877.png,Normal/images/Normal-6877.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6877.png,Normal/masks/Normal-6877.png,True,False,299,299,L,112.28,937367bcc3c0e0bbb27c440065bc2ef6
Normal-6878.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6878.png,Normal/images/Normal-6878.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6878.png,Normal/masks/Normal-6878.png,True,False,299,299,L,144.21,def56d9e752b996544dec4a5098fd4fc
Normal-6879.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6879.png,Normal/images/Normal-6879.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6879.png,Normal/masks/Normal-6879.png,True,False,299,299,L,158.65,3d12d2d07adbf7a8e02707c8df483bef
Normal-688.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-688.png,Normal/images/Normal-688.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-688.png,Normal/masks/Normal-688.png,True,False,299,299,L,116.06,226fb69b69cd411cdd8f0754462ba035
Normal-6880.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6880.png,Normal/images/Normal-6880.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6880.png,Normal/masks/Normal-6880.png,True,False,299,299,L,108.86,c7bf36edf42c6327aa339a91a01a5f95
Normal-6881.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6881.png,Normal/images/Normal-6881.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6881.png,Normal/masks/Normal-6881.png,True,False,299,299,L,124.09,52c693402e3ed5eb23f109c12c8dc5f6
Normal-6882.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6882.png,Normal/images/Normal-6882.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6882.png,Normal/masks/Normal-6882.png,True,False,299,299,L,94.02,73079dc4a03307671a7d1168bebb9bf6
Normal-6883.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6883.png,Normal/images/Normal-6883.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6883.png,Normal/masks/Normal-6883.png,True,False,299,299,L,122.63,d2cd7c11d1b1ebe6a561d6e904b93412
Normal-6884.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6884.png,Normal/images/Normal-6884.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6884.png,Normal/masks/Normal-6884.png,True,False,299,299,L,109.28,9fc26c53dbad50c53a8bdbee07cf0a77
Normal-6885.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6885.png,Normal/images/Normal-6885.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6885.png,Normal/masks/Normal-6885.png,True,False,299,299,L,128.92,eae6010492cae3f965181595d0f79685
Normal-6886.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6886.png,Normal/images/Normal-6886.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6886.png,Normal/masks/Normal-6886.png,True,False,299,299,L,117.53,676a2c463c7f80a7bd55ae8203edc00c
Normal-6887.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6887.png,Normal/images/Normal-6887.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6887.png,Normal/masks/Normal-6887.png,True,False,299,299,L,111.6,130aa341b0ddd9beece0583cdd2f4338
Normal-6888.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6888.png,Normal/images/Normal-6888.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6888.png,Normal/masks/Normal-6888.png,True,False,299,299,L,143.25,5dbf3a74a611d41215b37b4fa35a6ef8
Normal-6889.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6889.png,Normal/images/Normal-6889.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6889.png,Normal/masks/Normal-6889.png,True,False,299,299,L,134.55,a107541ec8cd92e0a295afc86484b15b
Normal-689.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-689.png,Normal/images/Normal-689.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-689.png,Normal/masks/Normal-689.png,True,False,299,299,L,131.7,2eb38564931f73a2551edb22ef7ecaa8
Normal-6890.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6890.png,Normal/images/Normal-6890.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6890.png,Normal/masks/Normal-6890.png,True,False,299,299,L,115.84,6481ffee0f8cc374d40493d289b0252e
Normal-6891.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6891.png,Normal/images/Normal-6891.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6891.png,Normal/masks/Normal-6891.png,True,False,299,299,L,130.45,c409d28b96ef030c02b3fe8dcadd2745
Normal-6892.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6892.png,Normal/images/Normal-6892.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6892.png,Normal/masks/Normal-6892.png,True,False,299,299,L,109.53,c86b1239b2bd52362862930c79762a48
Normal-6893.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6893.png,Normal/images/Normal-6893.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6893.png,Normal/masks/Normal-6893.png,True,False,299,299,L,116.45,959a039e996cd3b072a5e87f6dba44b0
Normal-6894.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6894.png,Normal/images/Normal-6894.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6894.png,Normal/masks/Normal-6894.png,True,False,299,299,L,153.32,894d6cdc24108d7f5a99df94d679a2ca
Normal-6895.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6895.png,Normal/images/Normal-6895.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6895.png,Normal/masks/Normal-6895.png,True,False,299,299,L,166.09,a200fe13d6167e6402b873e649b37254
Normal-6896.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6896.png,Normal/images/Normal-6896.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6896.png,Normal/masks/Normal-6896.png,True,False,299,299,L,166.74,f2c2773655e760235b44cdf12ad7be8b
Normal-6897.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6897.png,Normal/images/Normal-6897.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6897.png,Normal/masks/Normal-6897.png,True,False,299,299,L,119.91,734910886492bc99844bc22d80528739
Normal-6898.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6898.png,Normal/images/Normal-6898.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6898.png,Normal/masks/Normal-6898.png,True,False,299,299,L,128.8,a0d25c93ac8510bc1d18e135053dfc42
Normal-6899.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6899.png,Normal/images/Normal-6899.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6899.png,Normal/masks/Normal-6899.png,True,False,299,299,L,130.74,565d1afc383e21f328ac889458c214de
Normal-69.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-69.png,Normal/images/Normal-69.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-69.png,Normal/masks/Normal-69.png,True,False,299,299,L,97.21,9b54fb8fa70a5f106ea431ac02c30b3d
Normal-690.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-690.png,Normal/images/Normal-690.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-690.png,Normal/masks/Normal-690.png,True,False,299,299,L,124.89,31bfb86c7c7604d0a36457a4ff7d1155
Normal-6900.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6900.png,Normal/images/Normal-6900.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6900.png,Normal/masks/Normal-6900.png,True,False,299,299,L,122.02,d3a909960edfa7e1cd2651b539a079f1
Normal-6901.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6901.png,Normal/images/Normal-6901.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6901.png,Normal/masks/Normal-6901.png,True,False,299,299,L,128.84,f27a9b4c6b6d1caca166ce29f3f5ff64
Normal-6902.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6902.png,Normal/images/Normal-6902.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6902.png,Normal/masks/Normal-6902.png,True,False,299,299,L,122.51,624522953699fa21a46b597e1c86d9f1
Normal-6903.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6903.png,Normal/images/Normal-6903.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6903.png,Normal/masks/Normal-6903.png,True,False,299,299,L,115.11,83f6823f1441d2e1171a441c122edb2c
Normal-6904.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6904.png,Normal/images/Normal-6904.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6904.png,Normal/masks/Normal-6904.png,True,False,299,299,L,122.93,f52da9338271326ca758cc82d3881739
Normal-6905.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6905.png,Normal/images/Normal-6905.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6905.png,Normal/masks/Normal-6905.png,True,False,299,299,L,142.39,489102675a5071e2bb08648151faf5c7
Normal-6906.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6906.png,Normal/images/Normal-6906.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6906.png,Normal/masks/Normal-6906.png,True,False,299,299,L,123.35,9eb80f84b0c370442e324398b2ad7a39
Normal-6907.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6907.png,Normal/images/Normal-6907.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6907.png,Normal/masks/Normal-6907.png,True,False,299,299,L,122.04,02d39d77c1b53cdbbeb52e8163566d9c
Normal-6908.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6908.png,Normal/images/Normal-6908.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6908.png,Normal/masks/Normal-6908.png,True,False,299,299,L,172.47,2bcd549dba7e15e2adec28f56260c607
Normal-6909.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6909.png,Normal/images/Normal-6909.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6909.png,Normal/masks/Normal-6909.png,True,False,299,299,L,123.83,d67eb67d21ffd15494c5f401e156f3d1
Normal-691.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-691.png,Normal/images/Normal-691.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-691.png,Normal/masks/Normal-691.png,True,False,299,299,L,136.54,dda6a1ca0878367c60253d3b11343b76
Normal-6910.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6910.png,Normal/images/Normal-6910.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6910.png,Normal/masks/Normal-6910.png,True,False,299,299,L,166.35,e86b813538e747967874c2b47a9c3463
Normal-6911.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6911.png,Normal/images/Normal-6911.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6911.png,Normal/masks/Normal-6911.png,True,False,299,299,L,128.87,41ed8f5adbe7027e6cddbf5324096c1d
Normal-6912.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6912.png,Normal/images/Normal-6912.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6912.png,Normal/masks/Normal-6912.png,True,False,299,299,L,115.09,4acb4a9f827329f9922d87e19be13295
Normal-6913.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6913.png,Normal/images/Normal-6913.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6913.png,Normal/masks/Normal-6913.png,True,False,299,299,L,103.71,0c95a9a9d3484e2402aeec516d794514
Normal-6914.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6914.png,Normal/images/Normal-6914.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6914.png,Normal/masks/Normal-6914.png,True,False,299,299,L,160.01,4780bac32c874d3e96698c3c253e0f58
Normal-6915.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6915.png,Normal/images/Normal-6915.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6915.png,Normal/masks/Normal-6915.png,True,False,299,299,L,144.13,41f687f001189509bf3afec6d389db4c
Normal-6916.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6916.png,Normal/images/Normal-6916.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6916.png,Normal/masks/Normal-6916.png,True,False,299,299,L,167.46,5692dd53c255180f53c537911dde6d5f
Normal-6917.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6917.png,Normal/images/Normal-6917.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6917.png,Normal/masks/Normal-6917.png,True,False,299,299,L,117.8,e7732424b9431007d69c64e5010c79dd
Normal-6918.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6918.png,Normal/images/Normal-6918.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6918.png,Normal/masks/Normal-6918.png,True,False,299,299,L,111.89,d0659fbb4cbc716a22741d778be2f3bc
Normal-6919.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6919.png,Normal/images/Normal-6919.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6919.png,Normal/masks/Normal-6919.png,True,False,299,299,L,125.01,27e41bd7f00cf6db39c9d61d6cd4ce98
Normal-692.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-692.png,Normal/images/Normal-692.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-692.png,Normal/masks/Normal-692.png,True,False,299,299,L,116.39,1eecbba1c7cc34f8a0f9ed35144969c6
Normal-6920.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6920.png,Normal/images/Normal-6920.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6920.png,Normal/masks/Normal-6920.png,True,False,299,299,L,119.97,b89071862792505a6bd5e88ecbff3d68
Normal-6921.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6921.png,Normal/images/Normal-6921.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6921.png,Normal/masks/Normal-6921.png,True,False,299,299,L,110.45,0caf21c7f81dc71978241adbe6f3103a
Normal-6922.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6922.png,Normal/images/Normal-6922.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6922.png,Normal/masks/Normal-6922.png,True,False,299,299,L,107.81,4c64c7877f5b184e3c08f45ae23aa8a6
Normal-6923.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6923.png,Normal/images/Normal-6923.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6923.png,Normal/masks/Normal-6923.png,True,False,299,299,L,134.6,a3e27c0832befd577c88ae317d790679
Normal-6924.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6924.png,Normal/images/Normal-6924.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6924.png,Normal/masks/Normal-6924.png,True,False,299,299,L,114.66,84454d98418bfe42506e509c3eee463a
Normal-6925.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6925.png,Normal/images/Normal-6925.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6925.png,Normal/masks/Normal-6925.png,True,False,299,299,L,165.35,99e700504ec3c330ecbf492283125067
Normal-6926.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6926.png,Normal/images/Normal-6926.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6926.png,Normal/masks/Normal-6926.png,True,False,299,299,L,139.25,11eacc71d74f616cfddb06634f61a474
Normal-6927.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6927.png,Normal/images/Normal-6927.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6927.png,Normal/masks/Normal-6927.png,True,False,299,299,L,108.26,e1dd14c99dfa741681c059f55cdb38d7
Normal-6928.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6928.png,Normal/images/Normal-6928.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6928.png,Normal/masks/Normal-6928.png,True,False,299,299,L,150.41,e9c01210ad4d99abfd99301d143748ac
Normal-6929.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6929.png,Normal/images/Normal-6929.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6929.png,Normal/masks/Normal-6929.png,True,False,299,299,L,159.2,17da3ca0f1799bb469039ad7ce67ffdc
Normal-693.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-693.png,Normal/images/Normal-693.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-693.png,Normal/masks/Normal-693.png,True,False,299,299,L,112.18,650e52fc0496a3435b8a57be7dcf559c
Normal-6930.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6930.png,Normal/images/Normal-6930.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6930.png,Normal/masks/Normal-6930.png,True,False,299,299,L,118.29,8117c5867964f9ce25023deb248680e9
Normal-6931.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6931.png,Normal/images/Normal-6931.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6931.png,Normal/masks/Normal-6931.png,True,False,299,299,L,129.96,ff06a5e4c197c3cbd9c59efd8702f063
Normal-6932.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6932.png,Normal/images/Normal-6932.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6932.png,Normal/masks/Normal-6932.png,True,False,299,299,L,168.2,1e079b5a9bb879ca320e0705f5588752
Normal-6933.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6933.png,Normal/images/Normal-6933.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6933.png,Normal/masks/Normal-6933.png,True,False,299,299,L,106.79,a57d4fd15cf1f736ae4ea56d271eb2b4
Normal-6934.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6934.png,Normal/images/Normal-6934.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6934.png,Normal/masks/Normal-6934.png,True,False,299,299,L,161.43,be15e8daace0a22e9b322cc590fbeca4
Normal-6935.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6935.png,Normal/images/Normal-6935.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6935.png,Normal/masks/Normal-6935.png,True,False,299,299,L,113.0,7186c979cc8946267e13032beadff6e2
Normal-6936.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6936.png,Normal/images/Normal-6936.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6936.png,Normal/masks/Normal-6936.png,True,False,299,299,L,97.46,c5f12bfc7420e1d08515688a62d5bcbb
Normal-6937.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6937.png,Normal/images/Normal-6937.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6937.png,Normal/masks/Normal-6937.png,True,False,299,299,L,158.41,e57caa1b5a802b63a9c882131924493d
Normal-6938.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6938.png,Normal/images/Normal-6938.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6938.png,Normal/masks/Normal-6938.png,True,False,299,299,L,144.02,e5b68908d509507f6542be8055e03193
Normal-6939.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6939.png,Normal/images/Normal-6939.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6939.png,Normal/masks/Normal-6939.png,True,False,299,299,L,161.7,9e5318f1a0804c469dc445f0c7eaa3f8
Normal-694.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-694.png,Normal/images/Normal-694.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-694.png,Normal/masks/Normal-694.png,True,False,299,299,L,103.34,180899d333271950bef48523fd9751ec
Normal-6940.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6940.png,Normal/images/Normal-6940.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6940.png,Normal/masks/Normal-6940.png,True,False,299,299,L,112.28,6413a4e811695741a4bcf5e33a72f8f7
Normal-6941.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6941.png,Normal/images/Normal-6941.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6941.png,Normal/masks/Normal-6941.png,True,False,299,299,L,80.46,a6ff0063fbeface5abc74400367213bc
Normal-6942.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6942.png,Normal/images/Normal-6942.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6942.png,Normal/masks/Normal-6942.png,True,False,299,299,L,116.31,a15c1e09577e713da26380e5ca70a816
Normal-6943.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6943.png,Normal/images/Normal-6943.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6943.png,Normal/masks/Normal-6943.png,True,False,299,299,L,173.36,36b8e075652b9197ac6a386c90398f0b
Normal-6944.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6944.png,Normal/images/Normal-6944.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6944.png,Normal/masks/Normal-6944.png,True,False,299,299,L,127.31,242a8d86070f42c45d9d2a5a6b1cee35
Normal-6945.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6945.png,Normal/images/Normal-6945.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6945.png,Normal/masks/Normal-6945.png,True,False,299,299,L,86.55,a6b6ff44268596f8b59f889a65339b22
Normal-6946.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6946.png,Normal/images/Normal-6946.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6946.png,Normal/masks/Normal-6946.png,True,False,299,299,L,120.23,49e5c5e0f442c79d531a54d162e904e3
Normal-6947.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6947.png,Normal/images/Normal-6947.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6947.png,Normal/masks/Normal-6947.png,True,False,299,299,L,146.12,548286371a69557d3a70243f2f737183
Normal-6948.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6948.png,Normal/images/Normal-6948.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6948.png,Normal/masks/Normal-6948.png,True,False,299,299,L,128.95,2f0804a700545e1ce3786a7a3588d138
Normal-6949.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6949.png,Normal/images/Normal-6949.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6949.png,Normal/masks/Normal-6949.png,True,False,299,299,L,136.85,4771b6e84ea52b83f78090b2bfe4079f
Normal-695.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-695.png,Normal/images/Normal-695.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-695.png,Normal/masks/Normal-695.png,True,False,299,299,L,92.86,3a896b85c04fedf9665cea11fb8ba171
Normal-6950.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6950.png,Normal/images/Normal-6950.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6950.png,Normal/masks/Normal-6950.png,True,False,299,299,L,139.58,ed3a938a299cfd110ee1210fb2eeba74
Normal-6951.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6951.png,Normal/images/Normal-6951.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6951.png,Normal/masks/Normal-6951.png,True,False,299,299,L,70.74,eb0ee2cce833eb3890ee40db121eb7fd
Normal-6952.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6952.png,Normal/images/Normal-6952.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6952.png,Normal/masks/Normal-6952.png,True,False,299,299,L,104.45,3be40538d1f332c228771a36f28daae6
Normal-6953.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6953.png,Normal/images/Normal-6953.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6953.png,Normal/masks/Normal-6953.png,True,False,299,299,L,174.02,3fbe769b235a0d8e5ae5854c7a055f4a
Normal-6954.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6954.png,Normal/images/Normal-6954.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6954.png,Normal/masks/Normal-6954.png,True,False,299,299,L,102.08,656a17e9ec910af3e89492bee3662b3e
Normal-6955.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6955.png,Normal/images/Normal-6955.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6955.png,Normal/masks/Normal-6955.png,True,False,299,299,L,122.92,34b440132b8109c8f5c303889a68f95a
Normal-6956.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6956.png,Normal/images/Normal-6956.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6956.png,Normal/masks/Normal-6956.png,True,False,299,299,L,103.38,0b12f085f92818e39087ecbbe0ba95c9
Normal-6957.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6957.png,Normal/images/Normal-6957.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6957.png,Normal/masks/Normal-6957.png,True,False,299,299,L,117.78,e1fa969fc87752b1317064a0d84ad61c
Normal-6958.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6958.png,Normal/images/Normal-6958.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6958.png,Normal/masks/Normal-6958.png,True,False,299,299,L,111.86,8dabc0c98238fbdb2feee022c3c1439f
Normal-6959.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6959.png,Normal/images/Normal-6959.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6959.png,Normal/masks/Normal-6959.png,True,False,299,299,L,102.85,3c3bf3e3de802c46d45242fa4acb190b
Normal-696.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-696.png,Normal/images/Normal-696.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-696.png,Normal/masks/Normal-696.png,True,False,299,299,L,134.7,a4da5125fe2a1ba9b0de282a94f1b470
Normal-6960.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6960.png,Normal/images/Normal-6960.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6960.png,Normal/masks/Normal-6960.png,True,False,299,299,L,115.27,c6b2c5f2a8ad5b529a36e65642da03dc
Normal-6961.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6961.png,Normal/images/Normal-6961.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6961.png,Normal/masks/Normal-6961.png,True,False,299,299,L,122.8,62ebdcdd5e8e8f9d2009a8b6c761832f
Normal-6962.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6962.png,Normal/images/Normal-6962.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6962.png,Normal/masks/Normal-6962.png,True,False,299,299,L,121.79,bd39b5812d18807b7d6725e5143d5c0a
Normal-6963.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6963.png,Normal/images/Normal-6963.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6963.png,Normal/masks/Normal-6963.png,True,False,299,299,L,99.07,0234a45ec9f016e3061167e2f4307dc7
Normal-6964.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6964.png,Normal/images/Normal-6964.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6964.png,Normal/masks/Normal-6964.png,True,False,299,299,L,151.69,cf03c03212f87e17b55170a4ad8bb05d
Normal-6965.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6965.png,Normal/images/Normal-6965.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6965.png,Normal/masks/Normal-6965.png,True,False,299,299,L,105.27,6104c0df9ab8da321901dbbb9b0baf98
Normal-6966.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6966.png,Normal/images/Normal-6966.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6966.png,Normal/masks/Normal-6966.png,True,False,299,299,L,126.23,454f5f2722689d90218edcde87ebadea
Normal-6967.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6967.png,Normal/images/Normal-6967.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6967.png,Normal/masks/Normal-6967.png,True,False,299,299,L,131.39,6b7e31ce0b21c518297a4b3e6959a018
Normal-6968.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6968.png,Normal/images/Normal-6968.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6968.png,Normal/masks/Normal-6968.png,True,False,299,299,L,145.88,2153b8a7b80bc86a9ee0b69d14f1e09b
Normal-6969.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6969.png,Normal/images/Normal-6969.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6969.png,Normal/masks/Normal-6969.png,True,False,299,299,L,115.46,b2a93aa57b26921509fc63777c0b8b08
Normal-697.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-697.png,Normal/images/Normal-697.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-697.png,Normal/masks/Normal-697.png,True,False,299,299,L,122.96,567aa165c63069c7855ca4c027e7770a
Normal-6970.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6970.png,Normal/images/Normal-6970.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6970.png,Normal/masks/Normal-6970.png,True,False,299,299,L,109.21,b14383c305af9f438b47384dd9531d34
Normal-6971.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6971.png,Normal/images/Normal-6971.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6971.png,Normal/masks/Normal-6971.png,True,False,299,299,L,143.08,2be974615bade4389763523a391e0643
Normal-6972.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6972.png,Normal/images/Normal-6972.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6972.png,Normal/masks/Normal-6972.png,True,False,299,299,L,122.6,0f7eeeeed909d10add7179b9a4064ae1
Normal-6973.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6973.png,Normal/images/Normal-6973.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6973.png,Normal/masks/Normal-6973.png,True,False,299,299,L,154.38,ee35a6ed898dc2112768b671c34a3ce1
Normal-6974.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6974.png,Normal/images/Normal-6974.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6974.png,Normal/masks/Normal-6974.png,True,False,299,299,L,153.96,41da1cb7dd32a4a5c9da4398d60d2d39
Normal-6975.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6975.png,Normal/images/Normal-6975.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6975.png,Normal/masks/Normal-6975.png,True,False,299,299,L,113.08,bc61d3ffe65e757ed66be145e54d5c51
Normal-6976.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6976.png,Normal/images/Normal-6976.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6976.png,Normal/masks/Normal-6976.png,True,False,299,299,L,122.78,dd4978a4b41ad5fbf30bc4f832705016
Normal-6977.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6977.png,Normal/images/Normal-6977.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6977.png,Normal/masks/Normal-6977.png,True,False,299,299,L,112.02,f12fc89a0928bdf6ab3a932ed4068ac6
Normal-6978.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6978.png,Normal/images/Normal-6978.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6978.png,Normal/masks/Normal-6978.png,True,False,299,299,L,185.56,2d623c6939b04cd2a00416c6578185c2
Normal-6979.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6979.png,Normal/images/Normal-6979.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6979.png,Normal/masks/Normal-6979.png,True,False,299,299,L,107.07,1225c764d060b225b58c0c01c3ab55ab
Normal-698.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-698.png,Normal/images/Normal-698.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-698.png,Normal/masks/Normal-698.png,True,False,299,299,L,130.73,b4298af14a0df20ef0c2936519727f40
Normal-6980.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6980.png,Normal/images/Normal-6980.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6980.png,Normal/masks/Normal-6980.png,True,False,299,299,L,101.36,034028e3323a4bd7a0de5910382ac063
Normal-6981.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6981.png,Normal/images/Normal-6981.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6981.png,Normal/masks/Normal-6981.png,True,False,299,299,L,139.72,445bcfc607c58fedd618ecd62ae7e645
Normal-6982.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6982.png,Normal/images/Normal-6982.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6982.png,Normal/masks/Normal-6982.png,True,False,299,299,L,114.88,3265baa4028fcbf38644a0c25c14aa31
Normal-6983.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6983.png,Normal/images/Normal-6983.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6983.png,Normal/masks/Normal-6983.png,True,False,299,299,L,126.06,e4d04c71382813b06765b2e4dffb206b
Normal-6984.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6984.png,Normal/images/Normal-6984.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6984.png,Normal/masks/Normal-6984.png,True,False,299,299,L,165.43,eb42c8871fe1a2cc9a0c7f785ef44750
Normal-6985.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6985.png,Normal/images/Normal-6985.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6985.png,Normal/masks/Normal-6985.png,True,False,299,299,L,166.33,bec897f62d81a932b6d2b4ab534a339c
Normal-6986.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6986.png,Normal/images/Normal-6986.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6986.png,Normal/masks/Normal-6986.png,True,False,299,299,L,113.9,71bdf7bc19942bcbb4d9ef63c41b8a45
Normal-6987.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6987.png,Normal/images/Normal-6987.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6987.png,Normal/masks/Normal-6987.png,True,False,299,299,L,134.16,6549b81b5d29d98449118d37b35b7c9e
Normal-6988.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6988.png,Normal/images/Normal-6988.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6988.png,Normal/masks/Normal-6988.png,True,False,299,299,L,167.23,1d47c55ca22d8b8cda5a7b2dd4599d46
Normal-6989.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6989.png,Normal/images/Normal-6989.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6989.png,Normal/masks/Normal-6989.png,True,False,299,299,L,127.13,012ca7bd9c5049c795056cdd7d5690ac
Normal-699.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-699.png,Normal/images/Normal-699.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-699.png,Normal/masks/Normal-699.png,True,False,299,299,L,137.43,6ea89fed79a90ca160d511eb4d8c34f9
Normal-6990.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6990.png,Normal/images/Normal-6990.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6990.png,Normal/masks/Normal-6990.png,True,False,299,299,L,169.45,e4ee949eaa7fd1f5cb55648a6c390fea
Normal-6991.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6991.png,Normal/images/Normal-6991.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6991.png,Normal/masks/Normal-6991.png,True,False,299,299,L,111.71,c792e7bba43ab158aeb5c98cfdca604b
Normal-6992.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6992.png,Normal/images/Normal-6992.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6992.png,Normal/masks/Normal-6992.png,True,False,299,299,L,120.64,9cff21b6392d3c1fe813035c6bfe2551
Normal-6993.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6993.png,Normal/images/Normal-6993.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6993.png,Normal/masks/Normal-6993.png,True,False,299,299,L,164.54,06086d213b6d3aa45044d26975534055
Normal-6994.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6994.png,Normal/images/Normal-6994.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6994.png,Normal/masks/Normal-6994.png,True,False,299,299,L,99.39,d3650ea94a9f97078e7038ebe7a704bc
Normal-6995.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6995.png,Normal/images/Normal-6995.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6995.png,Normal/masks/Normal-6995.png,True,False,299,299,L,117.02,360f9853b35c1e67308826f17db54434
Normal-6996.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6996.png,Normal/images/Normal-6996.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6996.png,Normal/masks/Normal-6996.png,True,False,299,299,L,115.82,6598b4011e55207fa864ea40b2d03f5c
Normal-6997.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6997.png,Normal/images/Normal-6997.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6997.png,Normal/masks/Normal-6997.png,True,False,299,299,L,109.45,1bba42b2054f6d18dc6e35d8d124ada3
Normal-6998.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6998.png,Normal/images/Normal-6998.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6998.png,Normal/masks/Normal-6998.png,True,False,299,299,L,116.54,c8fc7f63c3ca36f6f6c59de7ec127d0f
Normal-6999.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-6999.png,Normal/images/Normal-6999.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-6999.png,Normal/masks/Normal-6999.png,True,False,299,299,L,146.06,6ba4b1f9a43fb189bef24d856261c432
Normal-7.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7.png,Normal/images/Normal-7.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7.png,Normal/masks/Normal-7.png,True,False,299,299,L,131.78,900850d442ddd08283cf567ba0b02778
Normal-70.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-70.png,Normal/images/Normal-70.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-70.png,Normal/masks/Normal-70.png,True,False,299,299,L,115.64,f5f0a8375f055ba4996ede84070b5f1f
Normal-700.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-700.png,Normal/images/Normal-700.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-700.png,Normal/masks/Normal-700.png,True,False,299,299,L,124.84,1f5408ad487fa4c9525530b650aaaa8c
Normal-7000.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7000.png,Normal/images/Normal-7000.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7000.png,Normal/masks/Normal-7000.png,True,False,299,299,L,118.87,8f3ff2d48f4b38ba807a78413f74cc5c
Normal-7001.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7001.png,Normal/images/Normal-7001.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7001.png,Normal/masks/Normal-7001.png,True,False,299,299,L,135.0,ace5ed8285542425caba4cf7405e4d77
Normal-7002.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7002.png,Normal/images/Normal-7002.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7002.png,Normal/masks/Normal-7002.png,True,False,299,299,L,100.84,4272b6d5ac87cada8510b6ec23e86c37
Normal-7003.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7003.png,Normal/images/Normal-7003.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7003.png,Normal/masks/Normal-7003.png,True,False,299,299,L,102.08,e8ef48d31806105d89ac4eec25257f81
Normal-7004.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7004.png,Normal/images/Normal-7004.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7004.png,Normal/masks/Normal-7004.png,True,False,299,299,L,120.51,fc321a5b6f7c3a94a5050a36a1b1ec5e
Normal-7005.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7005.png,Normal/images/Normal-7005.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7005.png,Normal/masks/Normal-7005.png,True,False,299,299,L,99.68,b9fdb1d6b57cc8a1020b758a7dc9a4ea
Normal-7006.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7006.png,Normal/images/Normal-7006.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7006.png,Normal/masks/Normal-7006.png,True,False,299,299,L,174.11,e62dc63615b21acff943d328579a80f5
Normal-7007.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7007.png,Normal/images/Normal-7007.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7007.png,Normal/masks/Normal-7007.png,True,False,299,299,L,130.59,4557ffb82ea7273a483dfe20c7902e8f
Normal-7008.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7008.png,Normal/images/Normal-7008.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7008.png,Normal/masks/Normal-7008.png,True,False,299,299,L,126.44,798596f91f3bddccc3217ed0aa512803
Normal-7009.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7009.png,Normal/images/Normal-7009.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7009.png,Normal/masks/Normal-7009.png,True,False,299,299,L,141.38,941c69cd1febcb5f91ae7d763455b604
Normal-701.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-701.png,Normal/images/Normal-701.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-701.png,Normal/masks/Normal-701.png,True,False,299,299,L,107.9,cfc81a594aa64a77b551f157bf1b332a
Normal-7010.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7010.png,Normal/images/Normal-7010.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7010.png,Normal/masks/Normal-7010.png,True,False,299,299,L,149.48,0780658e35cea88f8b1757f385f80734
Normal-7011.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7011.png,Normal/images/Normal-7011.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7011.png,Normal/masks/Normal-7011.png,True,False,299,299,L,127.36,b919ae4e1ce2024d079a1e3d5036feeb
Normal-7012.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7012.png,Normal/images/Normal-7012.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7012.png,Normal/masks/Normal-7012.png,True,False,299,299,L,129.04,a1820e8af0c5687925f3b0a00dbd9150
Normal-7013.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7013.png,Normal/images/Normal-7013.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7013.png,Normal/masks/Normal-7013.png,True,False,299,299,L,129.96,6093db1f56d0b7340a417350c1e17d3b
Normal-7014.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7014.png,Normal/images/Normal-7014.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7014.png,Normal/masks/Normal-7014.png,True,False,299,299,L,159.13,7a511ec4adbdc7ed778c9b080a132741
Normal-7015.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7015.png,Normal/images/Normal-7015.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7015.png,Normal/masks/Normal-7015.png,True,False,299,299,L,129.21,c4e943d1605ef9976d16a325d15242fd
Normal-7016.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7016.png,Normal/images/Normal-7016.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7016.png,Normal/masks/Normal-7016.png,True,False,299,299,L,114.64,264ca2f06ae141e98e7f215f87fbbd1d
Normal-7017.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7017.png,Normal/images/Normal-7017.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7017.png,Normal/masks/Normal-7017.png,True,False,299,299,L,123.68,7fddd6cdd43ee04ca8265e04a4a48cd7
Normal-7018.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7018.png,Normal/images/Normal-7018.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7018.png,Normal/masks/Normal-7018.png,True,False,299,299,L,149.87,b2de91cbbdb8ade3d9d79375fb9a5788
Normal-7019.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7019.png,Normal/images/Normal-7019.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7019.png,Normal/masks/Normal-7019.png,True,False,299,299,L,129.28,c3870ae1421cf932ff6bc441aeef397d
Normal-702.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-702.png,Normal/images/Normal-702.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-702.png,Normal/masks/Normal-702.png,True,False,299,299,L,154.79,3f20f2b6e71847019ae7f7d8037d101b
Normal-7020.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7020.png,Normal/images/Normal-7020.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7020.png,Normal/masks/Normal-7020.png,True,False,299,299,L,131.09,4550e453b78a93a9012553582ca993d0
Normal-7021.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7021.png,Normal/images/Normal-7021.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7021.png,Normal/masks/Normal-7021.png,True,False,299,299,L,127.92,d1f6b017777521fb9cc294a4bdadd3d0
Normal-7022.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7022.png,Normal/images/Normal-7022.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7022.png,Normal/masks/Normal-7022.png,True,False,299,299,L,124.24,2ca2a4774cb42ab083002f395e729acb
Normal-7023.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7023.png,Normal/images/Normal-7023.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7023.png,Normal/masks/Normal-7023.png,True,False,299,299,L,104.65,f0d6de605d68695131d0c703982854a9
Normal-7024.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7024.png,Normal/images/Normal-7024.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7024.png,Normal/masks/Normal-7024.png,True,False,299,299,L,114.95,9f929aeeb27d5cd508c8eb7cd6090bc5
Normal-7025.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7025.png,Normal/images/Normal-7025.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7025.png,Normal/masks/Normal-7025.png,True,False,299,299,L,134.43,904b78de315c37dfbbb8fd93d22342d2
Normal-7026.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7026.png,Normal/images/Normal-7026.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7026.png,Normal/masks/Normal-7026.png,True,False,299,299,L,133.72,309f7a3c385aeeefdd2d40fe43801f26
Normal-7027.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7027.png,Normal/images/Normal-7027.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7027.png,Normal/masks/Normal-7027.png,True,False,299,299,L,117.05,4cf6e25b123e6e6782e772e832ef7913
Normal-7028.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7028.png,Normal/images/Normal-7028.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7028.png,Normal/masks/Normal-7028.png,True,False,299,299,L,177.26,f390645a0dca939a6fdda4082abaffab
Normal-7029.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7029.png,Normal/images/Normal-7029.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7029.png,Normal/masks/Normal-7029.png,True,False,299,299,L,115.39,13824fed316e03217d0a0258ff6af529
Normal-703.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-703.png,Normal/images/Normal-703.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-703.png,Normal/masks/Normal-703.png,True,False,299,299,L,118.59,8a411cd3170ad0ad42a51f7cd7c4ca91
Normal-7030.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7030.png,Normal/images/Normal-7030.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7030.png,Normal/masks/Normal-7030.png,True,False,299,299,L,123.29,5f4f83dcbf754340bce63d7561543926
Normal-7031.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7031.png,Normal/images/Normal-7031.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7031.png,Normal/masks/Normal-7031.png,True,False,299,299,L,166.31,6ee5e8e81df3fed28824f89de1ed0771
Normal-7032.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7032.png,Normal/images/Normal-7032.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7032.png,Normal/masks/Normal-7032.png,True,False,299,299,L,137.92,067af712ad5834629ad5fd54bc90b162
Normal-7033.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7033.png,Normal/images/Normal-7033.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7033.png,Normal/masks/Normal-7033.png,True,False,299,299,L,127.58,d5b6eab65d517b47548f2fbd0c7dcabd
Normal-7034.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7034.png,Normal/images/Normal-7034.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7034.png,Normal/masks/Normal-7034.png,True,False,299,299,L,159.29,943c825342cdc5cad7e08c1e42013a83
Normal-7035.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7035.png,Normal/images/Normal-7035.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7035.png,Normal/masks/Normal-7035.png,True,False,299,299,L,125.63,55273dfcad820d592e3db2fb9a07129d
Normal-7036.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7036.png,Normal/images/Normal-7036.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7036.png,Normal/masks/Normal-7036.png,True,False,299,299,L,139.58,6d1fc30637a3095129fdfec8af7fd0f4
Normal-7037.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7037.png,Normal/images/Normal-7037.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7037.png,Normal/masks/Normal-7037.png,True,False,299,299,L,139.53,96966c40105c9f166d4cbc17c37b9198
Normal-7038.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7038.png,Normal/images/Normal-7038.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7038.png,Normal/masks/Normal-7038.png,True,False,299,299,L,117.76,3472fc57145f531742bbe406bea17823
Normal-7039.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7039.png,Normal/images/Normal-7039.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7039.png,Normal/masks/Normal-7039.png,True,False,299,299,L,123.21,05dece80241bdbd325df7cc8df06a805
Normal-704.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-704.png,Normal/images/Normal-704.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-704.png,Normal/masks/Normal-704.png,True,False,299,299,L,151.91,7427bf7d691cbc62998cac0f674bdced
Normal-7040.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7040.png,Normal/images/Normal-7040.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7040.png,Normal/masks/Normal-7040.png,True,False,299,299,L,101.34,b45f4dda1c0c7871a5590de72208976d
Normal-7041.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7041.png,Normal/images/Normal-7041.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7041.png,Normal/masks/Normal-7041.png,True,False,299,299,L,117.24,da3dfe5c72ad75eac89cd3aec017d649
Normal-7042.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7042.png,Normal/images/Normal-7042.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7042.png,Normal/masks/Normal-7042.png,True,False,299,299,L,146.61,1364297fe8f266cbba87cc8f05dc2275
Normal-7043.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7043.png,Normal/images/Normal-7043.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7043.png,Normal/masks/Normal-7043.png,True,False,299,299,L,93.67,20d387a9b189be83c07f137f46103f53
Normal-7044.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7044.png,Normal/images/Normal-7044.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7044.png,Normal/masks/Normal-7044.png,True,False,299,299,L,79.78,dc8bce098216382d85cc6b5625879b08
Normal-7045.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7045.png,Normal/images/Normal-7045.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7045.png,Normal/masks/Normal-7045.png,True,False,299,299,L,153.04,8f92398db910a1d1f4a88b57a81ee144
Normal-7046.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7046.png,Normal/images/Normal-7046.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7046.png,Normal/masks/Normal-7046.png,True,False,299,299,L,101.36,1e7f05f2605f65394dc00166e1b7a800
Normal-7047.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7047.png,Normal/images/Normal-7047.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7047.png,Normal/masks/Normal-7047.png,True,False,299,299,L,114.88,2482c38806117fa82db8b67fc4401a43
Normal-7048.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7048.png,Normal/images/Normal-7048.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7048.png,Normal/masks/Normal-7048.png,True,False,299,299,L,121.17,40edfd48da65a063f867e8fd60ffd4a1
Normal-7049.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7049.png,Normal/images/Normal-7049.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7049.png,Normal/masks/Normal-7049.png,True,False,299,299,L,170.78,0975954137093890ba54e9e3748b2776
Normal-705.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-705.png,Normal/images/Normal-705.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-705.png,Normal/masks/Normal-705.png,True,False,299,299,L,109.72,b7a8e94657eda497a47a018e82733fc2
Normal-7050.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7050.png,Normal/images/Normal-7050.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7050.png,Normal/masks/Normal-7050.png,True,False,299,299,L,176.24,41a223b8bb0e646421c3d6ebb6b8fe3f
Normal-7051.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7051.png,Normal/images/Normal-7051.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7051.png,Normal/masks/Normal-7051.png,True,False,299,299,L,116.29,2a0a7ce0499d5108634a8ab03ce65ff2
Normal-7052.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7052.png,Normal/images/Normal-7052.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7052.png,Normal/masks/Normal-7052.png,True,False,299,299,L,124.82,7d809576f8eb55ec342747d158da9bde
Normal-7053.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7053.png,Normal/images/Normal-7053.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7053.png,Normal/masks/Normal-7053.png,True,False,299,299,L,101.34,a67b5933d53ba3d78a598db7f496878a
Normal-7054.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7054.png,Normal/images/Normal-7054.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7054.png,Normal/masks/Normal-7054.png,True,False,299,299,L,147.99,25d8ff9291ee153c3b33f19186043299
Normal-7055.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7055.png,Normal/images/Normal-7055.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7055.png,Normal/masks/Normal-7055.png,True,False,299,299,L,121.68,70c85d85d629c6daddb40232ee3a2691
Normal-7056.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7056.png,Normal/images/Normal-7056.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7056.png,Normal/masks/Normal-7056.png,True,False,299,299,L,115.67,6627796ffd195aec0c5b1cc5af0a2ab7
Normal-7057.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7057.png,Normal/images/Normal-7057.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7057.png,Normal/masks/Normal-7057.png,True,False,299,299,L,106.81,78d4adc491ee7071c542791acff472e8
Normal-7058.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7058.png,Normal/images/Normal-7058.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7058.png,Normal/masks/Normal-7058.png,True,False,299,299,L,130.08,cf8316ae01d64ecfe4d9b26cb0e69512
Normal-7059.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7059.png,Normal/images/Normal-7059.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7059.png,Normal/masks/Normal-7059.png,True,False,299,299,L,120.46,9f4622db4ea3d6c14076ea0ef75cadbd
Normal-706.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-706.png,Normal/images/Normal-706.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-706.png,Normal/masks/Normal-706.png,True,False,299,299,L,133.11,55c10aa06dfdf0859df1e1feb236b179
Normal-7060.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7060.png,Normal/images/Normal-7060.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7060.png,Normal/masks/Normal-7060.png,True,False,299,299,L,114.42,a4075576600308f29b73e10e865cf9e6
Normal-7061.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7061.png,Normal/images/Normal-7061.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7061.png,Normal/masks/Normal-7061.png,True,False,299,299,L,115.55,25e4b1c3b91a3661e1efc2c005263f83
Normal-7062.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7062.png,Normal/images/Normal-7062.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7062.png,Normal/masks/Normal-7062.png,True,False,299,299,L,109.5,f8cbcc3d0e71da3c42514065e6a76166
Normal-7063.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7063.png,Normal/images/Normal-7063.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7063.png,Normal/masks/Normal-7063.png,True,False,299,299,L,154.22,7f6f199505481e1cdf2d107b8a7bf8cd
Normal-7064.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7064.png,Normal/images/Normal-7064.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7064.png,Normal/masks/Normal-7064.png,True,False,299,299,L,121.89,19727b601b80985353bebe5e0180db02
Normal-7065.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7065.png,Normal/images/Normal-7065.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7065.png,Normal/masks/Normal-7065.png,True,False,299,299,L,167.89,16d4138f5440c31d8d92bbc4a358a128
Normal-7066.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7066.png,Normal/images/Normal-7066.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7066.png,Normal/masks/Normal-7066.png,True,False,299,299,L,125.35,b78f5ea9d4f4fa81948138a454c33acd
Normal-7067.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7067.png,Normal/images/Normal-7067.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7067.png,Normal/masks/Normal-7067.png,True,False,299,299,L,171.98,7a925206e20606293891b800b8cd4993
Normal-7068.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7068.png,Normal/images/Normal-7068.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7068.png,Normal/masks/Normal-7068.png,True,False,299,299,L,125.89,4e3dbb5f334619d6907dd05b6a387ab1
Normal-7069.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7069.png,Normal/images/Normal-7069.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7069.png,Normal/masks/Normal-7069.png,True,False,299,299,L,156.5,618d99467f71f517f0e617b553fd4f03
Normal-707.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-707.png,Normal/images/Normal-707.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-707.png,Normal/masks/Normal-707.png,True,False,299,299,L,131.82,5814432aa1bbad737feab15d9814a3c4
Normal-7070.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7070.png,Normal/images/Normal-7070.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7070.png,Normal/masks/Normal-7070.png,True,False,299,299,L,116.63,703ac5f3f00617f3345f6f4142020f03
Normal-7071.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7071.png,Normal/images/Normal-7071.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7071.png,Normal/masks/Normal-7071.png,True,False,299,299,L,139.84,e2c07a9728bedd2d2cc9363d7df6c08c
Normal-7072.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7072.png,Normal/images/Normal-7072.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7072.png,Normal/masks/Normal-7072.png,True,False,299,299,L,106.83,dd2b041f6af2cbf8330863cb611d18a0
Normal-7073.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7073.png,Normal/images/Normal-7073.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7073.png,Normal/masks/Normal-7073.png,True,False,299,299,L,144.2,8556ad749d71ce865c9ffa7037d60f7b
Normal-7074.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7074.png,Normal/images/Normal-7074.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7074.png,Normal/masks/Normal-7074.png,True,False,299,299,L,128.97,59ca33029d84fba57c6676b56fc460b0
Normal-7075.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7075.png,Normal/images/Normal-7075.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7075.png,Normal/masks/Normal-7075.png,True,False,299,299,L,118.88,7470f17cd7cafef84680bd0768fddac2
Normal-7076.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7076.png,Normal/images/Normal-7076.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7076.png,Normal/masks/Normal-7076.png,True,False,299,299,L,116.17,17a25c28db29f2a8836b12354dba7ea6
Normal-7077.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7077.png,Normal/images/Normal-7077.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7077.png,Normal/masks/Normal-7077.png,True,False,299,299,L,118.99,e9e4e67c344e442a2ac2463b4330e4ea
Normal-7078.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7078.png,Normal/images/Normal-7078.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7078.png,Normal/masks/Normal-7078.png,True,False,299,299,L,127.42,89cf9cd323724c5e48921fd3a9f7862a
Normal-7079.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7079.png,Normal/images/Normal-7079.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7079.png,Normal/masks/Normal-7079.png,True,False,299,299,L,130.16,73b5139f341eb5e895509b21bfaeb5d8
Normal-708.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-708.png,Normal/images/Normal-708.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-708.png,Normal/masks/Normal-708.png,True,False,299,299,L,140.23,275855af741207b8874374545569b94a
Normal-7080.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7080.png,Normal/images/Normal-7080.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7080.png,Normal/masks/Normal-7080.png,True,False,299,299,L,135.52,6b2f495cf620d647010c45de564f4d56
Normal-7081.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7081.png,Normal/images/Normal-7081.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7081.png,Normal/masks/Normal-7081.png,True,False,299,299,L,124.52,abd529e3be552ea09a607f8c8256fa2d
Normal-7082.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7082.png,Normal/images/Normal-7082.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7082.png,Normal/masks/Normal-7082.png,True,False,299,299,L,122.96,3dc30a09e94d5690058711fa6e673761
Normal-7083.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7083.png,Normal/images/Normal-7083.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7083.png,Normal/masks/Normal-7083.png,True,False,299,299,L,117.74,9f17653d736b1d6569517b68a921f2f9
Normal-7084.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7084.png,Normal/images/Normal-7084.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7084.png,Normal/masks/Normal-7084.png,True,False,299,299,L,73.21,0526066c058af0294b19ee9bee796268
Normal-7085.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7085.png,Normal/images/Normal-7085.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7085.png,Normal/masks/Normal-7085.png,True,False,299,299,L,108.93,cddfd586ad1b366f9d50d27c69b4039d
Normal-7086.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7086.png,Normal/images/Normal-7086.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7086.png,Normal/masks/Normal-7086.png,True,False,299,299,L,158.39,ff973953a2e5c9ed5f161fc7004854e3
Normal-7087.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7087.png,Normal/images/Normal-7087.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7087.png,Normal/masks/Normal-7087.png,True,False,299,299,L,120.1,34c9e747df0797c196a14ad6e58cf3f5
Normal-7088.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7088.png,Normal/images/Normal-7088.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7088.png,Normal/masks/Normal-7088.png,True,False,299,299,L,144.96,6add5953b4b63c91726e927ec54dec44
Normal-7089.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7089.png,Normal/images/Normal-7089.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7089.png,Normal/masks/Normal-7089.png,True,False,299,299,L,113.92,3f554cb74d21598d1338b742cddeed11
Normal-709.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-709.png,Normal/images/Normal-709.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-709.png,Normal/masks/Normal-709.png,True,False,299,299,L,125.79,001eaee19bc9388b780e2dc125b626c2
Normal-7090.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7090.png,Normal/images/Normal-7090.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7090.png,Normal/masks/Normal-7090.png,True,False,299,299,L,120.76,d8027b7e56e45bd2d7ea3e0e4c6e4432
Normal-7091.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7091.png,Normal/images/Normal-7091.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7091.png,Normal/masks/Normal-7091.png,True,False,299,299,L,167.33,0c243b7eaab433a67310ae06620bb44b
Normal-7092.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7092.png,Normal/images/Normal-7092.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7092.png,Normal/masks/Normal-7092.png,True,False,299,299,L,88.25,fc046070ca9931581f4df6270ef71796
Normal-7093.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7093.png,Normal/images/Normal-7093.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7093.png,Normal/masks/Normal-7093.png,True,False,299,299,L,94.73,c086f1cb0a8309c86aaca328f4316268
Normal-7094.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7094.png,Normal/images/Normal-7094.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7094.png,Normal/masks/Normal-7094.png,True,False,299,299,L,128.01,3b6b8d6d5656251f21607ad87c5bca8d
Normal-7095.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7095.png,Normal/images/Normal-7095.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7095.png,Normal/masks/Normal-7095.png,True,False,299,299,L,117.95,2d69551b735883429a53a1b913b5c8cd
Normal-7096.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7096.png,Normal/images/Normal-7096.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7096.png,Normal/masks/Normal-7096.png,True,False,299,299,L,152.71,b36fc3fc58d21a1cb99b1f0e94382875
Normal-7097.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7097.png,Normal/images/Normal-7097.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7097.png,Normal/masks/Normal-7097.png,True,False,299,299,L,127.07,a4b8303eaa91f4ac6051f3a2abe4e701
Normal-7098.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7098.png,Normal/images/Normal-7098.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7098.png,Normal/masks/Normal-7098.png,True,False,299,299,L,123.43,86ef4b2afdaef9656fe8dec30dca38ce
Normal-7099.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7099.png,Normal/images/Normal-7099.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7099.png,Normal/masks/Normal-7099.png,True,False,299,299,L,132.8,9de6323375cbd351f08ef65abb4b5bac
Normal-71.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-71.png,Normal/images/Normal-71.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-71.png,Normal/masks/Normal-71.png,True,False,299,299,L,109.55,8252f2362001d6f6414c87ff79467620
Normal-710.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-710.png,Normal/images/Normal-710.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-710.png,Normal/masks/Normal-710.png,True,False,299,299,L,137.74,8d049687ffc08c7118ebb065bad56c16
Normal-7100.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7100.png,Normal/images/Normal-7100.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7100.png,Normal/masks/Normal-7100.png,True,False,299,299,L,139.87,2d76802e901d31e6a4739949cbdd2bdd
Normal-7101.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7101.png,Normal/images/Normal-7101.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7101.png,Normal/masks/Normal-7101.png,True,False,299,299,L,136.31,faf6185cd4eefa2a511e5c31cec3c051
Normal-7102.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7102.png,Normal/images/Normal-7102.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7102.png,Normal/masks/Normal-7102.png,True,False,299,299,L,161.58,93a65db3f770d4973aa186da2704c4f3
Normal-7103.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7103.png,Normal/images/Normal-7103.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7103.png,Normal/masks/Normal-7103.png,True,False,299,299,L,119.29,cf9d0e36370773527e8d9fc8712185fc
Normal-7104.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7104.png,Normal/images/Normal-7104.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7104.png,Normal/masks/Normal-7104.png,True,False,299,299,L,114.24,1694a8519beb798878bcfbfcc074a38e
Normal-7105.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7105.png,Normal/images/Normal-7105.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7105.png,Normal/masks/Normal-7105.png,True,False,299,299,L,167.48,1c9d13d027ae86a9202b89fe2991817d
Normal-7106.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7106.png,Normal/images/Normal-7106.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7106.png,Normal/masks/Normal-7106.png,True,False,299,299,L,135.27,b81ea0ef3a7ccf0483c8bbf1377d7c38
Normal-7107.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7107.png,Normal/images/Normal-7107.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7107.png,Normal/masks/Normal-7107.png,True,False,299,299,L,102.08,c8291cc82e1cabd0c7381f18008d2d65
Normal-7108.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7108.png,Normal/images/Normal-7108.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7108.png,Normal/masks/Normal-7108.png,True,False,299,299,L,126.48,4565051f3ef1386eaf3112f67e0037a6
Normal-7109.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7109.png,Normal/images/Normal-7109.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7109.png,Normal/masks/Normal-7109.png,True,False,299,299,L,111.73,96217682348334e06a49c10af35c0851
Normal-711.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-711.png,Normal/images/Normal-711.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-711.png,Normal/masks/Normal-711.png,True,False,299,299,L,123.55,2f2891d0621544dfb995c6f6662f0f70
Normal-7110.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7110.png,Normal/images/Normal-7110.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7110.png,Normal/masks/Normal-7110.png,True,False,299,299,L,116.08,1ee22528295c3b22f364f6e001bc9fb9
Normal-7111.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7111.png,Normal/images/Normal-7111.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7111.png,Normal/masks/Normal-7111.png,True,False,299,299,L,142.1,7bb609822d436ad0af2c34459d7fba83
Normal-7112.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7112.png,Normal/images/Normal-7112.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7112.png,Normal/masks/Normal-7112.png,True,False,299,299,L,145.45,496e461e2af4b1ca662f3f8916ff950a
Normal-7113.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7113.png,Normal/images/Normal-7113.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7113.png,Normal/masks/Normal-7113.png,True,False,299,299,L,165.05,41ee720d3f71dce24505eefca6d400ed
Normal-7114.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7114.png,Normal/images/Normal-7114.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7114.png,Normal/masks/Normal-7114.png,True,False,299,299,L,113.57,94b1b932d4ede198ec9838a67ba2f9ed
Normal-7115.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7115.png,Normal/images/Normal-7115.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7115.png,Normal/masks/Normal-7115.png,True,False,299,299,L,126.73,1853247727a0ad0b2dd0e6a31cbed7ac
Normal-7116.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7116.png,Normal/images/Normal-7116.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7116.png,Normal/masks/Normal-7116.png,True,False,299,299,L,142.05,425cfbd3255564553d05b3f86a366215
Normal-7117.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7117.png,Normal/images/Normal-7117.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7117.png,Normal/masks/Normal-7117.png,True,False,299,299,L,115.22,ec0933c48512396eabb3c83f37416244
Normal-7118.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7118.png,Normal/images/Normal-7118.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7118.png,Normal/masks/Normal-7118.png,True,False,299,299,L,74.89,c2061adb0e8e4403780b068ce4bb51e1
Normal-7119.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7119.png,Normal/images/Normal-7119.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7119.png,Normal/masks/Normal-7119.png,True,False,299,299,L,121.28,68877646f8e49ad449029d560e69b27d
Normal-712.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-712.png,Normal/images/Normal-712.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-712.png,Normal/masks/Normal-712.png,True,False,299,299,L,130.03,ff88a9428af97c98dda86204c07ca970
Normal-7120.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7120.png,Normal/images/Normal-7120.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7120.png,Normal/masks/Normal-7120.png,True,False,299,299,L,153.02,c14be37a17d8cd9fd0c024c4323c95e2
Normal-7121.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7121.png,Normal/images/Normal-7121.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7121.png,Normal/masks/Normal-7121.png,True,False,299,299,L,105.46,a55782019bb4ec1a3e794e6c1e8cc81f
Normal-7122.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7122.png,Normal/images/Normal-7122.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7122.png,Normal/masks/Normal-7122.png,True,False,299,299,L,116.58,cacd739d404e56009c08d0b1648ac52d
Normal-7123.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7123.png,Normal/images/Normal-7123.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7123.png,Normal/masks/Normal-7123.png,True,False,299,299,L,166.79,47fa6d4f530245d714bb6daed03275b2
Normal-7124.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7124.png,Normal/images/Normal-7124.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7124.png,Normal/masks/Normal-7124.png,True,False,299,299,L,104.25,6df170f65d53fa9431a11ab3b70531a2
Normal-7125.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7125.png,Normal/images/Normal-7125.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7125.png,Normal/masks/Normal-7125.png,True,False,299,299,L,106.51,9473ebd96932841778f613dda6bc71cb
Normal-7126.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7126.png,Normal/images/Normal-7126.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7126.png,Normal/masks/Normal-7126.png,True,False,299,299,L,98.49,da7b3bdc37e96b7f4ea6e52d54fd46bb
Normal-7127.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7127.png,Normal/images/Normal-7127.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7127.png,Normal/masks/Normal-7127.png,True,False,299,299,L,105.04,38baa783c732d16590f9993c0de803b3
Normal-7128.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7128.png,Normal/images/Normal-7128.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7128.png,Normal/masks/Normal-7128.png,True,False,299,299,L,115.35,8124a24e6b100ea3eec149d7ba6af70d
Normal-7129.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7129.png,Normal/images/Normal-7129.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7129.png,Normal/masks/Normal-7129.png,True,False,299,299,L,125.35,8ee018d11c6a625e841e8e8431ac53b4
Normal-713.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-713.png,Normal/images/Normal-713.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-713.png,Normal/masks/Normal-713.png,True,False,299,299,L,112.17,220f7987312789eaa37192299fbbaac4
Normal-7130.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7130.png,Normal/images/Normal-7130.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7130.png,Normal/masks/Normal-7130.png,True,False,299,299,L,109.88,24167f5eb176e5936e9afd4fb3ec3b7b
Normal-7131.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7131.png,Normal/images/Normal-7131.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7131.png,Normal/masks/Normal-7131.png,True,False,299,299,L,112.38,799bb1906513a3ca09d91734b8086273
Normal-7132.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7132.png,Normal/images/Normal-7132.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7132.png,Normal/masks/Normal-7132.png,True,False,299,299,L,142.79,4922f268465329704ef32acefd2120ec
Normal-7133.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7133.png,Normal/images/Normal-7133.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7133.png,Normal/masks/Normal-7133.png,True,False,299,299,L,165.04,7105f71ce3f3fc9223db2e3c53c6c81d
Normal-7134.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7134.png,Normal/images/Normal-7134.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7134.png,Normal/masks/Normal-7134.png,True,False,299,299,L,118.2,4676aed2e8681417828c43dd93e4c237
Normal-7135.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7135.png,Normal/images/Normal-7135.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7135.png,Normal/masks/Normal-7135.png,True,False,299,299,L,113.73,a804c35eb13ca49718805e0e2714e587
Normal-7136.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7136.png,Normal/images/Normal-7136.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7136.png,Normal/masks/Normal-7136.png,True,False,299,299,L,175.28,4e33c86ed6e71f26221bd562e48ada2d
Normal-7137.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7137.png,Normal/images/Normal-7137.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7137.png,Normal/masks/Normal-7137.png,True,False,299,299,L,130.0,96921d7848705e5b44dbbf4b2478e66d
Normal-7138.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7138.png,Normal/images/Normal-7138.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7138.png,Normal/masks/Normal-7138.png,True,False,299,299,L,173.72,85219910d31a6f322076385eb6f98c20
Normal-7139.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7139.png,Normal/images/Normal-7139.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7139.png,Normal/masks/Normal-7139.png,True,False,299,299,L,113.0,5f649121503cb55588536784f148929d
Normal-714.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-714.png,Normal/images/Normal-714.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-714.png,Normal/masks/Normal-714.png,True,False,299,299,L,119.33,47821b671826d9fcfc4e1c9c602d1327
Normal-7140.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7140.png,Normal/images/Normal-7140.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7140.png,Normal/masks/Normal-7140.png,True,False,299,299,L,142.81,29b637816ca08ca78cc0c8ce5edc0581
Normal-7141.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7141.png,Normal/images/Normal-7141.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7141.png,Normal/masks/Normal-7141.png,True,False,299,299,L,137.01,5133df3b5d7e53f955657547b2806242
Normal-7142.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7142.png,Normal/images/Normal-7142.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7142.png,Normal/masks/Normal-7142.png,True,False,299,299,L,110.2,4dfa44b19b02f0bdd04475349608c871
Normal-7143.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7143.png,Normal/images/Normal-7143.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7143.png,Normal/masks/Normal-7143.png,True,False,299,299,L,129.49,98668b2a7cb55327c2977d6c9f1a35b1
Normal-7144.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7144.png,Normal/images/Normal-7144.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7144.png,Normal/masks/Normal-7144.png,True,False,299,299,L,142.75,2b369c0cb7e72984f5f1e03b0b5081a8
Normal-7145.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7145.png,Normal/images/Normal-7145.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7145.png,Normal/masks/Normal-7145.png,True,False,299,299,L,132.31,a92c9b315d65fb78dccacc78c6eac52c
Normal-7146.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7146.png,Normal/images/Normal-7146.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7146.png,Normal/masks/Normal-7146.png,True,False,299,299,L,134.06,8bad13b66efff5c17681b6670f4a7014
Normal-7147.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7147.png,Normal/images/Normal-7147.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7147.png,Normal/masks/Normal-7147.png,True,False,299,299,L,105.08,a522e5b10e6a597c361c98e158f7f584
Normal-7148.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7148.png,Normal/images/Normal-7148.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7148.png,Normal/masks/Normal-7148.png,True,False,299,299,L,123.8,a820f0c8ed7c6e920fa6ff759c36a9f1
Normal-7149.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7149.png,Normal/images/Normal-7149.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7149.png,Normal/masks/Normal-7149.png,True,False,299,299,L,125.4,488bcd0eeddb26bac0b2db19fa0fdde5
Normal-715.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-715.png,Normal/images/Normal-715.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-715.png,Normal/masks/Normal-715.png,True,False,299,299,L,106.7,6337d458f67969c6b3b0a54068e7adeb
Normal-7150.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7150.png,Normal/images/Normal-7150.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7150.png,Normal/masks/Normal-7150.png,True,False,299,299,L,115.93,7fffe9bb2bc2371166aeb20daa2f3156
Normal-7151.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7151.png,Normal/images/Normal-7151.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7151.png,Normal/masks/Normal-7151.png,True,False,299,299,L,134.72,3c2e4c45754af23e295ff5a4de37eeea
Normal-7152.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7152.png,Normal/images/Normal-7152.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7152.png,Normal/masks/Normal-7152.png,True,False,299,299,L,141.07,1309966a88e0cae1cde832b0b919a513
Normal-7153.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7153.png,Normal/images/Normal-7153.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7153.png,Normal/masks/Normal-7153.png,True,False,299,299,L,114.92,abd30b11a258f8726129be74cf0bdad3
Normal-7154.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7154.png,Normal/images/Normal-7154.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7154.png,Normal/masks/Normal-7154.png,True,False,299,299,L,160.24,0941cc6ae6d969a7f09db5057ba4cc41
Normal-7155.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7155.png,Normal/images/Normal-7155.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7155.png,Normal/masks/Normal-7155.png,True,False,299,299,L,127.86,038f0d3cc120ed5ed950e72cd91b9f37
Normal-7156.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7156.png,Normal/images/Normal-7156.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7156.png,Normal/masks/Normal-7156.png,True,False,299,299,L,101.54,930a34f3bcc849b14c9efb4cec78f7cd
Normal-7157.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7157.png,Normal/images/Normal-7157.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7157.png,Normal/masks/Normal-7157.png,True,False,299,299,L,134.21,7e72c41fa56306f7aa846f2e171c24c2
Normal-7158.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7158.png,Normal/images/Normal-7158.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7158.png,Normal/masks/Normal-7158.png,True,False,299,299,L,128.34,02a4f3cca921ac921123a04412f41218
Normal-7159.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7159.png,Normal/images/Normal-7159.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7159.png,Normal/masks/Normal-7159.png,True,False,299,299,L,113.6,db1e54d3ce627d9beb89b56eb75ccf67
Normal-716.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-716.png,Normal/images/Normal-716.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-716.png,Normal/masks/Normal-716.png,True,False,299,299,L,124.84,70da1135db2bc6761067189daf97d875
Normal-7160.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7160.png,Normal/images/Normal-7160.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7160.png,Normal/masks/Normal-7160.png,True,False,299,299,L,139.15,8f8ec106a4535bc6d2235d755b22c6d9
Normal-7161.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7161.png,Normal/images/Normal-7161.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7161.png,Normal/masks/Normal-7161.png,True,False,299,299,L,156.63,e54f0e47068ce6bf999441b2f5f55ca0
Normal-7162.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7162.png,Normal/images/Normal-7162.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7162.png,Normal/masks/Normal-7162.png,True,False,299,299,L,160.58,01687da03945088968aa60725a0e88fe
Normal-7163.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7163.png,Normal/images/Normal-7163.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7163.png,Normal/masks/Normal-7163.png,True,False,299,299,L,162.47,33333ca51a8cf4d441756a99e6d10389
Normal-7164.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7164.png,Normal/images/Normal-7164.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7164.png,Normal/masks/Normal-7164.png,True,False,299,299,L,93.93,d1006a66dabce8f33e8e64ab18e301d1
Normal-7165.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7165.png,Normal/images/Normal-7165.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7165.png,Normal/masks/Normal-7165.png,True,False,299,299,L,144.83,931b3c168b8cb66a5faf2afcb8eb675b
Normal-7166.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7166.png,Normal/images/Normal-7166.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7166.png,Normal/masks/Normal-7166.png,True,False,299,299,L,164.87,1512dd1a2ad728c87484f7cb81bf56ee
Normal-7167.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7167.png,Normal/images/Normal-7167.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7167.png,Normal/masks/Normal-7167.png,True,False,299,299,L,107.75,cdae41c775f6fd94430ba6bc0849067b
Normal-7168.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7168.png,Normal/images/Normal-7168.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7168.png,Normal/masks/Normal-7168.png,True,False,299,299,L,124.5,dc14c9ec133ef5c7090dfd1b5fb91c57
Normal-7169.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7169.png,Normal/images/Normal-7169.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7169.png,Normal/masks/Normal-7169.png,True,False,299,299,L,119.05,6ceba851abe1760ef568830bbf3223b4
Normal-717.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-717.png,Normal/images/Normal-717.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-717.png,Normal/masks/Normal-717.png,True,False,299,299,L,111.66,47e4d6a96dacb5c2229046422ffc7296
Normal-7170.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7170.png,Normal/images/Normal-7170.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7170.png,Normal/masks/Normal-7170.png,True,False,299,299,L,164.8,de3d67543580f6ad1fb501aecca46c6c
Normal-7171.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7171.png,Normal/images/Normal-7171.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7171.png,Normal/masks/Normal-7171.png,True,False,299,299,L,138.55,3d00f4b4b28c775ecc0f08399f8ed566
Normal-7172.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7172.png,Normal/images/Normal-7172.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7172.png,Normal/masks/Normal-7172.png,True,False,299,299,L,91.81,2bfe31419fbd9b2b94e0e13cf57dc168
Normal-7173.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7173.png,Normal/images/Normal-7173.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7173.png,Normal/masks/Normal-7173.png,True,False,299,299,L,118.21,4e7393a3240579f55268401885940ff1
Normal-7174.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7174.png,Normal/images/Normal-7174.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7174.png,Normal/masks/Normal-7174.png,True,False,299,299,L,109.03,bdd4969fc45368944644d8e2a90b8e10
Normal-7175.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7175.png,Normal/images/Normal-7175.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7175.png,Normal/masks/Normal-7175.png,True,False,299,299,L,114.47,90aab6a14f8b110d8b570e0c29485525
Normal-7176.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7176.png,Normal/images/Normal-7176.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7176.png,Normal/masks/Normal-7176.png,True,False,299,299,L,172.52,a3ee300768ee6fb2fb95aeea03c8e2b9
Normal-7177.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7177.png,Normal/images/Normal-7177.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7177.png,Normal/masks/Normal-7177.png,True,False,299,299,L,120.07,f7d58983092eacefe15da15a4a4d519c
Normal-7178.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7178.png,Normal/images/Normal-7178.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7178.png,Normal/masks/Normal-7178.png,True,False,299,299,L,116.66,c8778fa4e13c975992932d90a931840e
Normal-7179.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7179.png,Normal/images/Normal-7179.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7179.png,Normal/masks/Normal-7179.png,True,False,299,299,L,141.41,62fcdc5061d226235be9bd2b4f6ce3ec
Normal-718.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-718.png,Normal/images/Normal-718.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-718.png,Normal/masks/Normal-718.png,True,False,299,299,L,121.72,dbd6368b54df0be274f14278dd155721
Normal-7180.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7180.png,Normal/images/Normal-7180.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7180.png,Normal/masks/Normal-7180.png,True,False,299,299,L,143.85,157b6e11584f6ed17bec5dee2c5230f5
Normal-7181.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7181.png,Normal/images/Normal-7181.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7181.png,Normal/masks/Normal-7181.png,True,False,299,299,L,126.27,d7ca49fc82a919072880d7a1a14caf1f
Normal-7182.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7182.png,Normal/images/Normal-7182.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7182.png,Normal/masks/Normal-7182.png,True,False,299,299,L,136.35,0eb5e82bf3af47cdcf5a56167d8c0f1e
Normal-7183.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7183.png,Normal/images/Normal-7183.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7183.png,Normal/masks/Normal-7183.png,True,False,299,299,L,114.26,6bd67464af4c2d59dcb4a505e43075a0
Normal-7184.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7184.png,Normal/images/Normal-7184.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7184.png,Normal/masks/Normal-7184.png,True,False,299,299,L,153.74,8b2792590498d23be23bbd0d3e48057c
Normal-7185.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7185.png,Normal/images/Normal-7185.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7185.png,Normal/masks/Normal-7185.png,True,False,299,299,L,121.28,b59a9b8a73989e89101b5ff8bc8d492d
Normal-7186.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7186.png,Normal/images/Normal-7186.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7186.png,Normal/masks/Normal-7186.png,True,False,299,299,L,148.79,a85e40a8906d75e4759e1b7f149e3f8c
Normal-7187.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7187.png,Normal/images/Normal-7187.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7187.png,Normal/masks/Normal-7187.png,True,False,299,299,L,153.31,929e5fadf4e6ca5fbbefda45d848b529
Normal-7188.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7188.png,Normal/images/Normal-7188.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7188.png,Normal/masks/Normal-7188.png,True,False,299,299,L,105.83,cf4b8eb865228f05796a95e3b7a156d7
Normal-7189.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7189.png,Normal/images/Normal-7189.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7189.png,Normal/masks/Normal-7189.png,True,False,299,299,L,137.28,58682415ddfacbfd1785cc06c4c008f3
Normal-719.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-719.png,Normal/images/Normal-719.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-719.png,Normal/masks/Normal-719.png,True,False,299,299,L,121.88,607035b28e2991313fd5aa13badd8396
Normal-7190.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7190.png,Normal/images/Normal-7190.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7190.png,Normal/masks/Normal-7190.png,True,False,299,299,L,116.58,e727cab4170bafa058fa522d18414eca
Normal-7191.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7191.png,Normal/images/Normal-7191.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7191.png,Normal/masks/Normal-7191.png,True,False,299,299,L,126.22,2abee56b71952e2e99b2e337dc287bbb
Normal-7192.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7192.png,Normal/images/Normal-7192.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7192.png,Normal/masks/Normal-7192.png,True,False,299,299,L,99.38,bd73e03ff50623c58bfc82b4ffc7e6c6
Normal-7193.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7193.png,Normal/images/Normal-7193.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7193.png,Normal/masks/Normal-7193.png,True,False,299,299,L,109.23,60f262496672d9eaaed36369deb7f71b
Normal-7194.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7194.png,Normal/images/Normal-7194.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7194.png,Normal/masks/Normal-7194.png,True,False,299,299,L,159.68,0d9461c12358ef4cf31547ec07783a56
Normal-7195.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7195.png,Normal/images/Normal-7195.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7195.png,Normal/masks/Normal-7195.png,True,False,299,299,L,154.96,bf0010f7c4099d69a6f50a50e7fb96ea
Normal-7196.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7196.png,Normal/images/Normal-7196.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7196.png,Normal/masks/Normal-7196.png,True,False,299,299,L,159.61,9e83c1f0a7d8e6b516f0f4865b0ac6ac
Normal-7197.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7197.png,Normal/images/Normal-7197.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7197.png,Normal/masks/Normal-7197.png,True,False,299,299,L,154.32,48d77cc1147f221168579cd4766edf05
Normal-7198.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7198.png,Normal/images/Normal-7198.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7198.png,Normal/masks/Normal-7198.png,True,False,299,299,L,131.25,8bc29ac4f38ba990aa89c1a3fea8c7c4
Normal-7199.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7199.png,Normal/images/Normal-7199.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7199.png,Normal/masks/Normal-7199.png,True,False,299,299,L,161.12,f0b6e7c5a3073b0396613bcbefc9b383
Normal-72.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-72.png,Normal/images/Normal-72.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-72.png,Normal/masks/Normal-72.png,True,False,299,299,L,109.62,3059ddbf13cddb6cc948d210527ec982
Normal-720.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-720.png,Normal/images/Normal-720.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-720.png,Normal/masks/Normal-720.png,True,False,299,299,L,112.31,43ab26058b13d4e9c90d0c8d8723d0c4
Normal-7200.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7200.png,Normal/images/Normal-7200.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7200.png,Normal/masks/Normal-7200.png,True,False,299,299,L,103.35,2acc3ef45191ff30b9304f9624eff1a9
Normal-7201.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7201.png,Normal/images/Normal-7201.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7201.png,Normal/masks/Normal-7201.png,True,False,299,299,L,134.12,002466a2e66a2a56721b6743a8f27e9f
Normal-7202.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7202.png,Normal/images/Normal-7202.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7202.png,Normal/masks/Normal-7202.png,True,False,299,299,L,120.35,c7d22edf21c55b73ba1c444fc9e4ad88
Normal-7203.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7203.png,Normal/images/Normal-7203.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7203.png,Normal/masks/Normal-7203.png,True,False,299,299,L,192.94,fa3dc61f16bdf03c811a06b17e6e53b9
Normal-7204.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7204.png,Normal/images/Normal-7204.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7204.png,Normal/masks/Normal-7204.png,True,False,299,299,L,178.74,1aa8d14aceef45602a705b4846ae7a2f
Normal-7205.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7205.png,Normal/images/Normal-7205.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7205.png,Normal/masks/Normal-7205.png,True,False,299,299,L,117.29,f35f0d0f14ede72456bcc225aab4a6aa
Normal-7206.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7206.png,Normal/images/Normal-7206.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7206.png,Normal/masks/Normal-7206.png,True,False,299,299,L,115.66,d70a33a6e8da56f109129809eb6eadcb
Normal-7207.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7207.png,Normal/images/Normal-7207.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7207.png,Normal/masks/Normal-7207.png,True,False,299,299,L,107.98,abfac52f1c72409fc1da7db743f8b1dc
Normal-7208.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7208.png,Normal/images/Normal-7208.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7208.png,Normal/masks/Normal-7208.png,True,False,299,299,L,121.06,4d7f3f9a5c2c9ff9126350552874a5d5
Normal-7209.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7209.png,Normal/images/Normal-7209.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7209.png,Normal/masks/Normal-7209.png,True,False,299,299,L,171.05,d264d2c582dfaac6be9d529bacb950b8
Normal-721.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-721.png,Normal/images/Normal-721.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-721.png,Normal/masks/Normal-721.png,True,False,299,299,L,100.98,d87808f6b8fbd29a8217140f81e20b45
Normal-7210.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7210.png,Normal/images/Normal-7210.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7210.png,Normal/masks/Normal-7210.png,True,False,299,299,L,103.38,d81419f3117b8732634c03d5dcf4dea7
Normal-7211.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7211.png,Normal/images/Normal-7211.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7211.png,Normal/masks/Normal-7211.png,True,False,299,299,L,148.61,b51e3b229d45ffa6d27f4489c3e1c574
Normal-7212.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7212.png,Normal/images/Normal-7212.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7212.png,Normal/masks/Normal-7212.png,True,False,299,299,L,166.26,5ebfa0b94709c5f9f8d59b93eb3703f1
Normal-7213.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7213.png,Normal/images/Normal-7213.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7213.png,Normal/masks/Normal-7213.png,True,False,299,299,L,142.28,31146a99df9508a8738bffd652bd8ff8
Normal-7214.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7214.png,Normal/images/Normal-7214.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7214.png,Normal/masks/Normal-7214.png,True,False,299,299,L,146.7,ceeced0fc7de18107bec9d17d7d5eb4e
Normal-7215.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7215.png,Normal/images/Normal-7215.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7215.png,Normal/masks/Normal-7215.png,True,False,299,299,L,132.06,524bb6df4cbefacbcfa518e5c55b27d1
Normal-7216.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7216.png,Normal/images/Normal-7216.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7216.png,Normal/masks/Normal-7216.png,True,False,299,299,L,102.81,b1b6431c616667306d871c17bd62838e
Normal-7217.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7217.png,Normal/images/Normal-7217.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7217.png,Normal/masks/Normal-7217.png,True,False,299,299,L,124.95,49d467417f2c90fa9a0818c3bd4328fb
Normal-7218.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7218.png,Normal/images/Normal-7218.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7218.png,Normal/masks/Normal-7218.png,True,False,299,299,L,167.75,d92373c12d1d948c6a032dbb2e1a6332
Normal-7219.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7219.png,Normal/images/Normal-7219.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7219.png,Normal/masks/Normal-7219.png,True,False,299,299,L,104.6,337719f0d63f29da3a53c2ae9636b079
Normal-722.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-722.png,Normal/images/Normal-722.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-722.png,Normal/masks/Normal-722.png,True,False,299,299,L,123.07,6a4e42d1d65def1b0d01df3fc2d155ce
Normal-7220.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7220.png,Normal/images/Normal-7220.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7220.png,Normal/masks/Normal-7220.png,True,False,299,299,L,150.81,1ea94d623a97d2c8f9e662ffa9054fc1
Normal-7221.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7221.png,Normal/images/Normal-7221.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7221.png,Normal/masks/Normal-7221.png,True,False,299,299,L,115.24,6d526dfcf2e96101ee0315a2b5e72f4e
Normal-7222.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7222.png,Normal/images/Normal-7222.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7222.png,Normal/masks/Normal-7222.png,True,False,299,299,L,121.56,88b9f64b3c68a12ba5d9dbced6c7414c
Normal-7223.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7223.png,Normal/images/Normal-7223.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7223.png,Normal/masks/Normal-7223.png,True,False,299,299,L,109.43,95638b9136c2177e569a3c58b470d174
Normal-7224.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7224.png,Normal/images/Normal-7224.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7224.png,Normal/masks/Normal-7224.png,True,False,299,299,L,164.63,04493d69e4372adcef8e3c5bb5a48fb9
Normal-7225.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7225.png,Normal/images/Normal-7225.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7225.png,Normal/masks/Normal-7225.png,True,False,299,299,L,127.21,8fb087cd920ac3e42e185b9cc52b6519
Normal-7226.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7226.png,Normal/images/Normal-7226.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7226.png,Normal/masks/Normal-7226.png,True,False,299,299,L,133.24,eaf3f1283eefbae20770aae709e66606
Normal-7227.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7227.png,Normal/images/Normal-7227.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7227.png,Normal/masks/Normal-7227.png,True,False,299,299,L,144.08,54facb0a725d88620298adf629eb6faa
Normal-7228.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7228.png,Normal/images/Normal-7228.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7228.png,Normal/masks/Normal-7228.png,True,False,299,299,L,133.1,44f18ca1574448e32297f6a32ea7dc79
Normal-7229.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7229.png,Normal/images/Normal-7229.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7229.png,Normal/masks/Normal-7229.png,True,False,299,299,L,178.51,a1bae0f356cfab7f3b8dda53966f30bc
Normal-723.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-723.png,Normal/images/Normal-723.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-723.png,Normal/masks/Normal-723.png,True,False,299,299,L,128.2,341b91db96b6635077c3cd682262229d
Normal-7230.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7230.png,Normal/images/Normal-7230.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7230.png,Normal/masks/Normal-7230.png,True,False,299,299,L,107.19,47ae49f15403604480cc3c0c9e702097
Normal-7231.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7231.png,Normal/images/Normal-7231.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7231.png,Normal/masks/Normal-7231.png,True,False,299,299,L,143.59,2892f2e622323eb57b093d0e2425942e
Normal-7232.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7232.png,Normal/images/Normal-7232.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7232.png,Normal/masks/Normal-7232.png,True,False,299,299,L,156.33,cea91b166e16d23a52c8a841379e0ec9
Normal-7233.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7233.png,Normal/images/Normal-7233.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7233.png,Normal/masks/Normal-7233.png,True,False,299,299,L,82.06,7fd677f95714cc28d2b758969c9bfc93
Normal-7234.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7234.png,Normal/images/Normal-7234.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7234.png,Normal/masks/Normal-7234.png,True,False,299,299,L,164.37,1400408ffabca1f935b60619805a835d
Normal-7235.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7235.png,Normal/images/Normal-7235.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7235.png,Normal/masks/Normal-7235.png,True,False,299,299,L,141.57,7bfe26b4bb9e2786214441c11d6310f2
Normal-7236.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7236.png,Normal/images/Normal-7236.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7236.png,Normal/masks/Normal-7236.png,True,False,299,299,L,176.84,08703bdad8ed2754a751845824be91ee
Normal-7237.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7237.png,Normal/images/Normal-7237.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7237.png,Normal/masks/Normal-7237.png,True,False,299,299,L,108.52,23a653e583cd888ca86cc904d31d2f0a
Normal-7238.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7238.png,Normal/images/Normal-7238.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7238.png,Normal/masks/Normal-7238.png,True,False,299,299,L,108.55,6f2176983c7328d776bb7a4ef26bae47
Normal-7239.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7239.png,Normal/images/Normal-7239.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7239.png,Normal/masks/Normal-7239.png,True,False,299,299,L,109.96,8e6cc733ea0c4f619fa2c9aac5308403
Normal-724.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-724.png,Normal/images/Normal-724.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-724.png,Normal/masks/Normal-724.png,True,False,299,299,L,123.23,ba00111e13f43d26c8beb037a86ee876
Normal-7240.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7240.png,Normal/images/Normal-7240.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7240.png,Normal/masks/Normal-7240.png,True,False,299,299,L,132.42,ef9435d84d7e3d7885b3f564fb85db25
Normal-7241.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7241.png,Normal/images/Normal-7241.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7241.png,Normal/masks/Normal-7241.png,True,False,299,299,L,167.09,2750cc771876b9e05b8740589b018394
Normal-7242.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7242.png,Normal/images/Normal-7242.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7242.png,Normal/masks/Normal-7242.png,True,False,299,299,L,124.66,2fe71aceb5c8932fd6d87bc7439805af
Normal-7243.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7243.png,Normal/images/Normal-7243.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7243.png,Normal/masks/Normal-7243.png,True,False,299,299,L,161.93,2c9fd5c4990d49483772f73483563292
Normal-7244.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7244.png,Normal/images/Normal-7244.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7244.png,Normal/masks/Normal-7244.png,True,False,299,299,L,139.4,2cd6d5396f5506d44c5db2bb58fdfa30
Normal-7245.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7245.png,Normal/images/Normal-7245.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7245.png,Normal/masks/Normal-7245.png,True,False,299,299,L,133.58,e28e9a773e8680bf3e2f669dadcd9d95
Normal-7246.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7246.png,Normal/images/Normal-7246.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7246.png,Normal/masks/Normal-7246.png,True,False,299,299,L,147.9,e2e546b58ae0a5b68bcab5b26f4c31da
Normal-7247.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7247.png,Normal/images/Normal-7247.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7247.png,Normal/masks/Normal-7247.png,True,False,299,299,L,127.73,6097b80d12b985a98e97b16e73b7652d
Normal-7248.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7248.png,Normal/images/Normal-7248.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7248.png,Normal/masks/Normal-7248.png,True,False,299,299,L,94.04,a49ed886bec1d2c4027c6f7d4bab6395
Normal-7249.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7249.png,Normal/images/Normal-7249.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7249.png,Normal/masks/Normal-7249.png,True,False,299,299,L,99.79,995200bd6be74f0c5a230a407be6589d
Normal-725.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-725.png,Normal/images/Normal-725.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-725.png,Normal/masks/Normal-725.png,True,False,299,299,L,124.9,14216a2bbde2997ac4ff67bdd6c84267
Normal-7250.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7250.png,Normal/images/Normal-7250.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7250.png,Normal/masks/Normal-7250.png,True,False,299,299,L,173.36,b086f486a65b74140ab5b4ac3ab5244c
Normal-7251.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7251.png,Normal/images/Normal-7251.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7251.png,Normal/masks/Normal-7251.png,True,False,299,299,L,122.82,2f0d9244cc7968e64c2c9ea6eceb4713
Normal-7252.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7252.png,Normal/images/Normal-7252.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7252.png,Normal/masks/Normal-7252.png,True,False,299,299,L,138.27,cd00ee2cf3cd6489a6184b2ec217a684
Normal-7253.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7253.png,Normal/images/Normal-7253.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7253.png,Normal/masks/Normal-7253.png,True,False,299,299,L,124.01,a1f204d5916756a03845680ca700fc86
Normal-7254.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7254.png,Normal/images/Normal-7254.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7254.png,Normal/masks/Normal-7254.png,True,False,299,299,L,104.01,e2af66db790ca1f92378e2fbbb50e679
Normal-7255.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7255.png,Normal/images/Normal-7255.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7255.png,Normal/masks/Normal-7255.png,True,False,299,299,L,171.62,66e5d24c8aab659091fbdbc78a0480a6
Normal-7256.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7256.png,Normal/images/Normal-7256.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7256.png,Normal/masks/Normal-7256.png,True,False,299,299,L,104.74,f5d8da7b73125f2fed04de461d9ac102
Normal-7257.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7257.png,Normal/images/Normal-7257.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7257.png,Normal/masks/Normal-7257.png,True,False,299,299,L,175.27,85bf928a294b51b3a0b6e7e372c10b93
Normal-7258.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7258.png,Normal/images/Normal-7258.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7258.png,Normal/masks/Normal-7258.png,True,False,299,299,L,125.65,b5351dff79b917cb161edb691f8f4db3
Normal-7259.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7259.png,Normal/images/Normal-7259.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7259.png,Normal/masks/Normal-7259.png,True,False,299,299,L,137.83,3a877142aa8bb307fbb130998988f1b7
Normal-726.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-726.png,Normal/images/Normal-726.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-726.png,Normal/masks/Normal-726.png,True,False,299,299,L,121.15,e8c745677b25cce295e6091cfe3094bf
Normal-7260.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7260.png,Normal/images/Normal-7260.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7260.png,Normal/masks/Normal-7260.png,True,False,299,299,L,176.6,c5a7f793d32fb83a7fd4c6d6ebc7b9ab
Normal-7261.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7261.png,Normal/images/Normal-7261.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7261.png,Normal/masks/Normal-7261.png,True,False,299,299,L,160.21,adedc0f71e40da03b91eeb3be63e7161
Normal-7262.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7262.png,Normal/images/Normal-7262.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7262.png,Normal/masks/Normal-7262.png,True,False,299,299,L,132.31,a9268bd39d63feba7b78ff5316613aba
Normal-7263.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7263.png,Normal/images/Normal-7263.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7263.png,Normal/masks/Normal-7263.png,True,False,299,299,L,104.36,0d22ef55a92c34ae864d4fde57965747
Normal-7264.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7264.png,Normal/images/Normal-7264.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7264.png,Normal/masks/Normal-7264.png,True,False,299,299,L,155.01,c17b9adaffb94dd09accd6d8a55beef0
Normal-7265.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7265.png,Normal/images/Normal-7265.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7265.png,Normal/masks/Normal-7265.png,True,False,299,299,L,107.71,baf3d514443e47ca4a8bc6468b7ac5ef
Normal-7266.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7266.png,Normal/images/Normal-7266.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7266.png,Normal/masks/Normal-7266.png,True,False,299,299,L,111.5,346a8b8fd6198f6d8104212a5f46087e
Normal-7267.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7267.png,Normal/images/Normal-7267.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7267.png,Normal/masks/Normal-7267.png,True,False,299,299,L,166.94,cc52a9b89d220173834d22bd2cba5704
Normal-7268.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7268.png,Normal/images/Normal-7268.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7268.png,Normal/masks/Normal-7268.png,True,False,299,299,L,108.57,215dafd277ecefe4d0bc97533f753977
Normal-7269.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7269.png,Normal/images/Normal-7269.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7269.png,Normal/masks/Normal-7269.png,True,False,299,299,L,123.55,a349d19c3ae6ca057e956d46cfd1a16f
Normal-727.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-727.png,Normal/images/Normal-727.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-727.png,Normal/masks/Normal-727.png,True,False,299,299,L,100.79,cf0065826145c591b39a2936104a9a4f
Normal-7270.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7270.png,Normal/images/Normal-7270.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7270.png,Normal/masks/Normal-7270.png,True,False,299,299,L,77.26,75478e54cc93af8e37928a0036fc48bc
Normal-7271.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7271.png,Normal/images/Normal-7271.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7271.png,Normal/masks/Normal-7271.png,True,False,299,299,L,170.1,c16522dcd2245fbb7fb3289c254362b7
Normal-7272.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7272.png,Normal/images/Normal-7272.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7272.png,Normal/masks/Normal-7272.png,True,False,299,299,L,121.97,257f5ce81d1a83f94d025ecb3f9f949d
Normal-7273.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7273.png,Normal/images/Normal-7273.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7273.png,Normal/masks/Normal-7273.png,True,False,299,299,L,162.5,3bc5eeaf12648909d033858dcba187d6
Normal-7274.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7274.png,Normal/images/Normal-7274.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7274.png,Normal/masks/Normal-7274.png,True,False,299,299,L,138.77,1222915934eb927c346163e4c9bbd1b1
Normal-7275.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7275.png,Normal/images/Normal-7275.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7275.png,Normal/masks/Normal-7275.png,True,False,299,299,L,106.66,20afb5c534dd4bc8feed9bbd429bff93
Normal-7276.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7276.png,Normal/images/Normal-7276.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7276.png,Normal/masks/Normal-7276.png,True,False,299,299,L,159.84,e8b387b5774360011715340df7f21e41
Normal-7277.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7277.png,Normal/images/Normal-7277.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7277.png,Normal/masks/Normal-7277.png,True,False,299,299,L,168.04,e7ee7b7091de8c3b07a65aedaad66330
Normal-7278.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7278.png,Normal/images/Normal-7278.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7278.png,Normal/masks/Normal-7278.png,True,False,299,299,L,108.82,71ceb865ffa905b240916872d4504061
Normal-7279.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7279.png,Normal/images/Normal-7279.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7279.png,Normal/masks/Normal-7279.png,True,False,299,299,L,116.3,dc8994b0bab6010947503842bf77a448
Normal-728.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-728.png,Normal/images/Normal-728.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-728.png,Normal/masks/Normal-728.png,True,False,299,299,L,124.32,8a5fa23b86854f8e9fd1c9bae249656b
Normal-7280.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7280.png,Normal/images/Normal-7280.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7280.png,Normal/masks/Normal-7280.png,True,False,299,299,L,91.07,aca6501b0981d932a6add7e64aea7d58
Normal-7281.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7281.png,Normal/images/Normal-7281.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7281.png,Normal/masks/Normal-7281.png,True,False,299,299,L,160.98,45f2c40144b5abcd038dbd2620c3e1e7
Normal-7282.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7282.png,Normal/images/Normal-7282.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7282.png,Normal/masks/Normal-7282.png,True,False,299,299,L,122.42,549d3d9f76552d70c20c66b8d5811797
Normal-7283.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7283.png,Normal/images/Normal-7283.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7283.png,Normal/masks/Normal-7283.png,True,False,299,299,L,127.28,ea1b8a43f12fc5ad3180c9270a1263be
Normal-7284.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7284.png,Normal/images/Normal-7284.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7284.png,Normal/masks/Normal-7284.png,True,False,299,299,L,114.82,377ad1cac199bc2039fd16fe98d7c641
Normal-7285.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7285.png,Normal/images/Normal-7285.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7285.png,Normal/masks/Normal-7285.png,True,False,299,299,L,99.87,178b6dab290eff6fd6d8982afc14ac77
Normal-7286.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7286.png,Normal/images/Normal-7286.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7286.png,Normal/masks/Normal-7286.png,True,False,299,299,L,145.17,b978e5bf6fa5c58cf1433ecc69120152
Normal-7287.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7287.png,Normal/images/Normal-7287.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7287.png,Normal/masks/Normal-7287.png,True,False,299,299,L,125.59,0efdaf3121261159689d3b2a31ce3de1
Normal-7288.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7288.png,Normal/images/Normal-7288.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7288.png,Normal/masks/Normal-7288.png,True,False,299,299,L,137.01,d1018aa9ea0ba6636c6a147b14878772
Normal-7289.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7289.png,Normal/images/Normal-7289.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7289.png,Normal/masks/Normal-7289.png,True,False,299,299,L,79.94,ee676c0abf44369930a5a7cf893a71a2
Normal-729.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-729.png,Normal/images/Normal-729.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-729.png,Normal/masks/Normal-729.png,True,False,299,299,L,123.26,47302829d16f71cba9ba5f41e6a43929
Normal-7290.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7290.png,Normal/images/Normal-7290.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7290.png,Normal/masks/Normal-7290.png,True,False,299,299,L,105.19,575249dab31be25dce9ee910a9a912f1
Normal-7291.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7291.png,Normal/images/Normal-7291.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7291.png,Normal/masks/Normal-7291.png,True,False,299,299,L,114.26,4c85ed83ce679935d42599cbbf1b15e2
Normal-7292.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7292.png,Normal/images/Normal-7292.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7292.png,Normal/masks/Normal-7292.png,True,False,299,299,L,112.8,b14ce745944ef1677db900b4ce7ed6fc
Normal-7293.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7293.png,Normal/images/Normal-7293.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7293.png,Normal/masks/Normal-7293.png,True,False,299,299,L,124.05,8d4e9b94ce5720c305592d9e4231372a
Normal-7294.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7294.png,Normal/images/Normal-7294.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7294.png,Normal/masks/Normal-7294.png,True,False,299,299,L,126.91,52594c0c86461892719cfa41f349d4c5
Normal-7295.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7295.png,Normal/images/Normal-7295.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7295.png,Normal/masks/Normal-7295.png,True,False,299,299,L,102.18,d743d352378f0035ffe8b2157af0c374
Normal-7296.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7296.png,Normal/images/Normal-7296.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7296.png,Normal/masks/Normal-7296.png,True,False,299,299,L,124.58,c95f17ec8cd6462490a67fefeaae1aa1
Normal-7297.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7297.png,Normal/images/Normal-7297.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7297.png,Normal/masks/Normal-7297.png,True,False,299,299,L,117.86,312107c090273a31027d2bda39e40506
Normal-7298.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7298.png,Normal/images/Normal-7298.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7298.png,Normal/masks/Normal-7298.png,True,False,299,299,L,121.53,79d60971a439b3af95d8de382627baf7
Normal-7299.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7299.png,Normal/images/Normal-7299.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7299.png,Normal/masks/Normal-7299.png,True,False,299,299,L,112.84,4dd34a3db9564d496a8f4c31a62f1f14
Normal-73.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-73.png,Normal/images/Normal-73.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-73.png,Normal/masks/Normal-73.png,True,False,299,299,L,120.3,c77bcf1c8b43aaaf45886accfeccd24e
Normal-730.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-730.png,Normal/images/Normal-730.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-730.png,Normal/masks/Normal-730.png,True,False,299,299,L,115.82,cae97bc010f7be81b537382e59a15c24
Normal-7300.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7300.png,Normal/images/Normal-7300.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7300.png,Normal/masks/Normal-7300.png,True,False,299,299,L,125.14,696480ce0ade5ac8332404e700d3f865
Normal-7301.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7301.png,Normal/images/Normal-7301.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7301.png,Normal/masks/Normal-7301.png,True,False,299,299,L,116.87,95f7d3b7ae13379efb85e6d463c2711c
Normal-7302.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7302.png,Normal/images/Normal-7302.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7302.png,Normal/masks/Normal-7302.png,True,False,299,299,L,82.9,8d584556f8a564cb4f8c7a3dbcf79e23
Normal-7303.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7303.png,Normal/images/Normal-7303.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7303.png,Normal/masks/Normal-7303.png,True,False,299,299,L,179.16,ba3bb2e0534aee5032857baf0cabcf64
Normal-7304.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7304.png,Normal/images/Normal-7304.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7304.png,Normal/masks/Normal-7304.png,True,False,299,299,L,114.74,ba39348ec29b5de62e39993400c5d850
Normal-7305.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7305.png,Normal/images/Normal-7305.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7305.png,Normal/masks/Normal-7305.png,True,False,299,299,L,168.02,70d8a0b6ee3505246083d76c7630d0df
Normal-7306.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7306.png,Normal/images/Normal-7306.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7306.png,Normal/masks/Normal-7306.png,True,False,299,299,L,135.31,135ed8b45af69f10b7b0bda6345eb97a
Normal-7307.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7307.png,Normal/images/Normal-7307.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7307.png,Normal/masks/Normal-7307.png,True,False,299,299,L,114.31,99370891d4c01a278d775e4c98e5678d
Normal-7308.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7308.png,Normal/images/Normal-7308.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7308.png,Normal/masks/Normal-7308.png,True,False,299,299,L,119.8,50e1e064080c8b916d7dc2c3b5bdf169
Normal-7309.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7309.png,Normal/images/Normal-7309.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7309.png,Normal/masks/Normal-7309.png,True,False,299,299,L,111.07,99803dc7270ecba6f0f3e925c18844ea
Normal-731.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-731.png,Normal/images/Normal-731.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-731.png,Normal/masks/Normal-731.png,True,False,299,299,L,139.38,66665a2c5bc2c22d209105dd4c6b78f3
Normal-7310.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7310.png,Normal/images/Normal-7310.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7310.png,Normal/masks/Normal-7310.png,True,False,299,299,L,129.97,749b88c540178d1f612f118b0d63b006
Normal-7311.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7311.png,Normal/images/Normal-7311.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7311.png,Normal/masks/Normal-7311.png,True,False,299,299,L,138.52,4ce0a443ed80483cdaa852b1e0852bd8
Normal-7312.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7312.png,Normal/images/Normal-7312.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7312.png,Normal/masks/Normal-7312.png,True,False,299,299,L,100.83,c54946b97ce78d6ca4de872de87a6944
Normal-7313.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7313.png,Normal/images/Normal-7313.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7313.png,Normal/masks/Normal-7313.png,True,False,299,299,L,124.81,82a21bb875dd9a128a9eabe8f7d6e7e9
Normal-7314.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7314.png,Normal/images/Normal-7314.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7314.png,Normal/masks/Normal-7314.png,True,False,299,299,L,137.16,4b6fe96b4b1e2d84b9fe4ea3d08cee6b
Normal-7315.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7315.png,Normal/images/Normal-7315.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7315.png,Normal/masks/Normal-7315.png,True,False,299,299,L,112.23,ffab433c34545acf480d4baa8dded337
Normal-7316.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7316.png,Normal/images/Normal-7316.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7316.png,Normal/masks/Normal-7316.png,True,False,299,299,L,140.56,32e3e04f367795edc3460f70b0eeefb0
Normal-7317.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7317.png,Normal/images/Normal-7317.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7317.png,Normal/masks/Normal-7317.png,True,False,299,299,L,108.45,aa7911ab748329483570f5cd7bc02b57
Normal-7318.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7318.png,Normal/images/Normal-7318.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7318.png,Normal/masks/Normal-7318.png,True,False,299,299,L,90.32,d1231f817d4ed81dc848d90a07d64e77
Normal-7319.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7319.png,Normal/images/Normal-7319.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7319.png,Normal/masks/Normal-7319.png,True,False,299,299,L,124.94,e4d78cb35a26171756a356f183022295
Normal-732.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-732.png,Normal/images/Normal-732.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-732.png,Normal/masks/Normal-732.png,True,False,299,299,L,139.74,e03c2b076c6b0157287ae60c24790295
Normal-7320.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7320.png,Normal/images/Normal-7320.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7320.png,Normal/masks/Normal-7320.png,True,False,299,299,L,119.69,db247b7f5db969722a1948471ef6c66c
Normal-7321.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7321.png,Normal/images/Normal-7321.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7321.png,Normal/masks/Normal-7321.png,True,False,299,299,L,138.81,209166d4ace21b8ea364e735349f7448
Normal-7322.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7322.png,Normal/images/Normal-7322.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7322.png,Normal/masks/Normal-7322.png,True,False,299,299,L,126.27,27e3d85c7be6d8c8e91c0226a588e530
Normal-7323.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7323.png,Normal/images/Normal-7323.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7323.png,Normal/masks/Normal-7323.png,True,False,299,299,L,123.71,38a14fd116dbbdb6a6b5467604e443a0
Normal-7324.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7324.png,Normal/images/Normal-7324.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7324.png,Normal/masks/Normal-7324.png,True,False,299,299,L,110.22,483fb223ddfcbeb32d0e777a7a9a9dad
Normal-7325.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7325.png,Normal/images/Normal-7325.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7325.png,Normal/masks/Normal-7325.png,True,False,299,299,L,108.31,3f567fd400c894e270e1800c01a433cb
Normal-7326.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7326.png,Normal/images/Normal-7326.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7326.png,Normal/masks/Normal-7326.png,True,False,299,299,L,104.54,1f00b88e501977c536bd753da012a688
Normal-7327.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7327.png,Normal/images/Normal-7327.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7327.png,Normal/masks/Normal-7327.png,True,False,299,299,L,139.11,19901d6ea208c8fcf55807242a11f33c
Normal-7328.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7328.png,Normal/images/Normal-7328.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7328.png,Normal/masks/Normal-7328.png,True,False,299,299,L,148.88,0966d3e39a718c8afaf0b12029812778
Normal-7329.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7329.png,Normal/images/Normal-7329.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7329.png,Normal/masks/Normal-7329.png,True,False,299,299,L,166.02,dc81c7dab0d7e15e6c1bbbc046577653
Normal-733.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-733.png,Normal/images/Normal-733.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-733.png,Normal/masks/Normal-733.png,True,False,299,299,L,125.01,d620de2c8056ef914f312ebe0789b219
Normal-7330.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7330.png,Normal/images/Normal-7330.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7330.png,Normal/masks/Normal-7330.png,True,False,299,299,L,154.01,25318638f76ac631c755c739cdae198b
Normal-7331.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7331.png,Normal/images/Normal-7331.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7331.png,Normal/masks/Normal-7331.png,True,False,299,299,L,104.68,83ef6ed59e7a5383c42fd8f0db406fb5
Normal-7332.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7332.png,Normal/images/Normal-7332.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7332.png,Normal/masks/Normal-7332.png,True,False,299,299,L,131.89,b32c984ff16e78f2f2d969819567096f
Normal-7333.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7333.png,Normal/images/Normal-7333.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7333.png,Normal/masks/Normal-7333.png,True,False,299,299,L,135.36,8befa908efc98c45e87cde211e491307
Normal-7334.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7334.png,Normal/images/Normal-7334.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7334.png,Normal/masks/Normal-7334.png,True,False,299,299,L,133.59,8fd8583ae4802562c92c123674fe290f
Normal-7335.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7335.png,Normal/images/Normal-7335.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7335.png,Normal/masks/Normal-7335.png,True,False,299,299,L,107.24,2ff81d85a5c4855ce9a74b2b04d37181
Normal-7336.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7336.png,Normal/images/Normal-7336.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7336.png,Normal/masks/Normal-7336.png,True,False,299,299,L,162.63,c99b6f03a323b32a0f11e824fc2b6435
Normal-7337.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7337.png,Normal/images/Normal-7337.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7337.png,Normal/masks/Normal-7337.png,True,False,299,299,L,147.68,0e6b03c0b386702f80c92316c032a686
Normal-7338.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7338.png,Normal/images/Normal-7338.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7338.png,Normal/masks/Normal-7338.png,True,False,299,299,L,158.69,5c2cfcf5e6b3e11430ee1aa65a1c87d1
Normal-7339.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7339.png,Normal/images/Normal-7339.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7339.png,Normal/masks/Normal-7339.png,True,False,299,299,L,113.26,f88f88848b570e973ed0d8336d210080
Normal-734.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-734.png,Normal/images/Normal-734.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-734.png,Normal/masks/Normal-734.png,True,False,299,299,L,113.78,1a296baf08c54cea73a9afc507b18c1b
Normal-7340.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7340.png,Normal/images/Normal-7340.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7340.png,Normal/masks/Normal-7340.png,True,False,299,299,L,114.9,0a9831c9d0d772c542b34642ef898413
Normal-7341.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7341.png,Normal/images/Normal-7341.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7341.png,Normal/masks/Normal-7341.png,True,False,299,299,L,137.83,80f0a31b1b4748475fec03f71557b775
Normal-7342.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7342.png,Normal/images/Normal-7342.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7342.png,Normal/masks/Normal-7342.png,True,False,299,299,L,170.12,386859b16ae2504d352e3dc77c7f0f9a
Normal-7343.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7343.png,Normal/images/Normal-7343.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7343.png,Normal/masks/Normal-7343.png,True,False,299,299,L,153.47,4cb058722c6d7023a52cade30927dd0d
Normal-7344.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7344.png,Normal/images/Normal-7344.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7344.png,Normal/masks/Normal-7344.png,True,False,299,299,L,137.81,6aba6e8ec81752674ef00a968221f01e
Normal-7345.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7345.png,Normal/images/Normal-7345.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7345.png,Normal/masks/Normal-7345.png,True,False,299,299,L,116.27,caad29f2d3695df7fdc62a19526ce45c
Normal-7346.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7346.png,Normal/images/Normal-7346.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7346.png,Normal/masks/Normal-7346.png,True,False,299,299,L,97.76,279a071d8bb6ee4ed323aee7c70be042
Normal-7347.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7347.png,Normal/images/Normal-7347.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7347.png,Normal/masks/Normal-7347.png,True,False,299,299,L,160.33,81c761d2e90559acc87a0feea08e3e9b
Normal-7348.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7348.png,Normal/images/Normal-7348.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7348.png,Normal/masks/Normal-7348.png,True,False,299,299,L,117.19,e32ebb0c6d97b53841f677fb5d1e245f
Normal-7349.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7349.png,Normal/images/Normal-7349.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7349.png,Normal/masks/Normal-7349.png,True,False,299,299,L,124.86,20870ee6b3e8b59b75a6d50e68919725
Normal-735.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-735.png,Normal/images/Normal-735.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-735.png,Normal/masks/Normal-735.png,True,False,299,299,L,109.04,d60e39fab3c675682ae8388cfd2ae1a0
Normal-7350.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7350.png,Normal/images/Normal-7350.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7350.png,Normal/masks/Normal-7350.png,True,False,299,299,L,170.32,89fee0b945de4a7cebdd3a5e9d78a6b5
Normal-7351.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7351.png,Normal/images/Normal-7351.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7351.png,Normal/masks/Normal-7351.png,True,False,299,299,L,159.77,60f51646fdc71d2b89be5fdeed3ad8a2
Normal-7352.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7352.png,Normal/images/Normal-7352.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7352.png,Normal/masks/Normal-7352.png,True,False,299,299,L,152.02,f632b3c73294421621346b7a024c052b
Normal-7353.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7353.png,Normal/images/Normal-7353.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7353.png,Normal/masks/Normal-7353.png,True,False,299,299,L,155.98,8a8d7d4e538bb15cf4e581a2fa25e06c
Normal-7354.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7354.png,Normal/images/Normal-7354.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7354.png,Normal/masks/Normal-7354.png,True,False,299,299,L,108.7,b7f70cf06830f9c3301e8b51d39d1387
Normal-7355.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7355.png,Normal/images/Normal-7355.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7355.png,Normal/masks/Normal-7355.png,True,False,299,299,L,104.41,41edd413ce0493b8a324fc2f35279af3
Normal-7356.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7356.png,Normal/images/Normal-7356.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7356.png,Normal/masks/Normal-7356.png,True,False,299,299,L,128.95,0c9ab7cdc34d114a82940563d59c6e37
Normal-7357.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7357.png,Normal/images/Normal-7357.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7357.png,Normal/masks/Normal-7357.png,True,False,299,299,L,131.33,a2b34bf67cf78cfc3f986810ba9bdaf1
Normal-7358.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7358.png,Normal/images/Normal-7358.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7358.png,Normal/masks/Normal-7358.png,True,False,299,299,L,124.65,cb7954c7e8da46dc15dcc5b6f510dbe2
Normal-7359.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7359.png,Normal/images/Normal-7359.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7359.png,Normal/masks/Normal-7359.png,True,False,299,299,L,133.49,4d27b8de4482a6e39c508f6b1da1d5f3
Normal-736.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-736.png,Normal/images/Normal-736.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-736.png,Normal/masks/Normal-736.png,True,False,299,299,L,141.07,08f6b45b9ddf3e79ab2e603c18e6a548
Normal-7360.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7360.png,Normal/images/Normal-7360.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7360.png,Normal/masks/Normal-7360.png,True,False,299,299,L,110.52,406772f12058e535ff66e78138958430
Normal-7361.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7361.png,Normal/images/Normal-7361.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7361.png,Normal/masks/Normal-7361.png,True,False,299,299,L,172.41,906cc00cff7114b193068a913df802d4
Normal-7362.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7362.png,Normal/images/Normal-7362.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7362.png,Normal/masks/Normal-7362.png,True,False,299,299,L,150.45,889b01c0a02aa3e22d8b3afa86febe30
Normal-7363.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7363.png,Normal/images/Normal-7363.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7363.png,Normal/masks/Normal-7363.png,True,False,299,299,L,124.93,0659054084744a6077e811c668f48656
Normal-7364.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7364.png,Normal/images/Normal-7364.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7364.png,Normal/masks/Normal-7364.png,True,False,299,299,L,141.01,a52a42b82373cf6635067924c3ff0d40
Normal-7365.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7365.png,Normal/images/Normal-7365.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7365.png,Normal/masks/Normal-7365.png,True,False,299,299,L,140.9,e1450f3b0c49a6aac3526db1ec656403
Normal-7366.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7366.png,Normal/images/Normal-7366.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7366.png,Normal/masks/Normal-7366.png,True,False,299,299,L,109.13,921944f5546bc4c235c530164afa20c4
Normal-7367.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7367.png,Normal/images/Normal-7367.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7367.png,Normal/masks/Normal-7367.png,True,False,299,299,L,113.71,5c76bac7121002dbda6c84d6c1098b3f
Normal-7368.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7368.png,Normal/images/Normal-7368.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7368.png,Normal/masks/Normal-7368.png,True,False,299,299,L,129.9,158abef23458669a5e1a6471426fd0b2
Normal-7369.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7369.png,Normal/images/Normal-7369.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7369.png,Normal/masks/Normal-7369.png,True,False,299,299,L,142.14,e3942575de16dd74e4171fd1d288986a
Normal-737.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-737.png,Normal/images/Normal-737.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-737.png,Normal/masks/Normal-737.png,True,False,299,299,L,131.18,f120bfd3ad188da18304afc682c1347c
Normal-7370.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7370.png,Normal/images/Normal-7370.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7370.png,Normal/masks/Normal-7370.png,True,False,299,299,L,125.01,bf41cf372d6c6b07b0bd4ec6d8419e7c
Normal-7371.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7371.png,Normal/images/Normal-7371.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7371.png,Normal/masks/Normal-7371.png,True,False,299,299,L,129.2,a325eef7e0734e20c6b113fe87a184ff
Normal-7372.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7372.png,Normal/images/Normal-7372.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7372.png,Normal/masks/Normal-7372.png,True,False,299,299,L,158.49,b7088465f2b5ec45515adfc5d221193c
Normal-7373.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7373.png,Normal/images/Normal-7373.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7373.png,Normal/masks/Normal-7373.png,True,False,299,299,L,166.18,52fed92bda53843fac8617c0fbf05a60
Normal-7374.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7374.png,Normal/images/Normal-7374.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7374.png,Normal/masks/Normal-7374.png,True,False,299,299,L,148.17,30133f7773236f4423c0b02da4a707d5
Normal-7375.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7375.png,Normal/images/Normal-7375.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7375.png,Normal/masks/Normal-7375.png,True,False,299,299,L,132.34,cec6b1bfa467a43880ec0261759ac87d
Normal-7376.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7376.png,Normal/images/Normal-7376.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7376.png,Normal/masks/Normal-7376.png,True,False,299,299,L,97.09,8ac100fbcab04e57ce5cc8ced7ddaa89
Normal-7377.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7377.png,Normal/images/Normal-7377.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7377.png,Normal/masks/Normal-7377.png,True,False,299,299,L,113.52,900e35054b48d429a45a1ce65b14e205
Normal-7378.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7378.png,Normal/images/Normal-7378.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7378.png,Normal/masks/Normal-7378.png,True,False,299,299,L,104.43,f161bcb68f4bebfdea0e12460f94d731
Normal-7379.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7379.png,Normal/images/Normal-7379.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7379.png,Normal/masks/Normal-7379.png,True,False,299,299,L,120.82,a7e7fb35ad2ad3394c1b705060e8deeb
Normal-738.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-738.png,Normal/images/Normal-738.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-738.png,Normal/masks/Normal-738.png,True,False,299,299,L,117.43,58103b93f00af26712f29980ff7ade11
Normal-7380.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7380.png,Normal/images/Normal-7380.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7380.png,Normal/masks/Normal-7380.png,True,False,299,299,L,141.52,695461e3ad8360ce1dcc7cdcf00a4f42
Normal-7381.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7381.png,Normal/images/Normal-7381.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7381.png,Normal/masks/Normal-7381.png,True,False,299,299,L,124.98,b9a3d23870a85b9508a55ef8e06c1e25
Normal-7382.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7382.png,Normal/images/Normal-7382.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7382.png,Normal/masks/Normal-7382.png,True,False,299,299,L,142.1,c43fcb0a6ea9fa2c017fa28e8eaf8db4
Normal-7383.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7383.png,Normal/images/Normal-7383.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7383.png,Normal/masks/Normal-7383.png,True,False,299,299,L,148.07,80a6f84a8771744fe22e0d29dbbd90b2
Normal-7384.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7384.png,Normal/images/Normal-7384.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7384.png,Normal/masks/Normal-7384.png,True,False,299,299,L,147.83,95ea6087b4e7fa51c4af044646f05bb9
Normal-7385.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7385.png,Normal/images/Normal-7385.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7385.png,Normal/masks/Normal-7385.png,True,False,299,299,L,135.44,bad3f8b8cd531657d48d21c6aae8d985
Normal-7386.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7386.png,Normal/images/Normal-7386.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7386.png,Normal/masks/Normal-7386.png,True,False,299,299,L,120.7,28809121aa833be9ee2fd5defa48cf64
Normal-7387.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7387.png,Normal/images/Normal-7387.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7387.png,Normal/masks/Normal-7387.png,True,False,299,299,L,105.22,c02f9c17ababb5dcd5774e41d138a55b
Normal-7388.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7388.png,Normal/images/Normal-7388.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7388.png,Normal/masks/Normal-7388.png,True,False,299,299,L,98.35,c803e91480599792c62739f56ed8f47b
Normal-7389.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7389.png,Normal/images/Normal-7389.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7389.png,Normal/masks/Normal-7389.png,True,False,299,299,L,104.44,9a4038e71b3b04ce1c47af30cee92387
Normal-739.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-739.png,Normal/images/Normal-739.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-739.png,Normal/masks/Normal-739.png,True,False,299,299,L,126.46,500aa7441d12148df98aa285df984d0f
Normal-7390.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7390.png,Normal/images/Normal-7390.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7390.png,Normal/masks/Normal-7390.png,True,False,299,299,L,127.61,4c50a56b4471c4333d56227b5b6be859
Normal-7391.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7391.png,Normal/images/Normal-7391.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7391.png,Normal/masks/Normal-7391.png,True,False,299,299,L,118.21,115eaab2d618f4f52aec0462026f3d9d
Normal-7392.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7392.png,Normal/images/Normal-7392.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7392.png,Normal/masks/Normal-7392.png,True,False,299,299,L,110.73,aa11adf9a75c5458bc6e687a025a7be9
Normal-7393.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7393.png,Normal/images/Normal-7393.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7393.png,Normal/masks/Normal-7393.png,True,False,299,299,L,160.84,4813937a61ee2abd17e7ac7049d3b369
Normal-7394.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7394.png,Normal/images/Normal-7394.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7394.png,Normal/masks/Normal-7394.png,True,False,299,299,L,117.54,f026ac258a64e8609f334d3c7d16adde
Normal-7395.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7395.png,Normal/images/Normal-7395.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7395.png,Normal/masks/Normal-7395.png,True,False,299,299,L,110.97,3f46887c64e58937d1edde2a6da493cb
Normal-7396.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7396.png,Normal/images/Normal-7396.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7396.png,Normal/masks/Normal-7396.png,True,False,299,299,L,139.34,cc9dbb535d4a664908965d627fc3452f
Normal-7397.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7397.png,Normal/images/Normal-7397.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7397.png,Normal/masks/Normal-7397.png,True,False,299,299,L,125.99,3081735c690e3bd84f2cebb1914a7a31
Normal-7398.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7398.png,Normal/images/Normal-7398.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7398.png,Normal/masks/Normal-7398.png,True,False,299,299,L,153.35,1b72f9718ccb878a79d87948a6849e68
Normal-7399.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7399.png,Normal/images/Normal-7399.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7399.png,Normal/masks/Normal-7399.png,True,False,299,299,L,141.55,c565aa0fe86ab4e804c284f5c61df4e2
Normal-74.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-74.png,Normal/images/Normal-74.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-74.png,Normal/masks/Normal-74.png,True,False,299,299,L,127.72,c9cb76e7691f209427035174833f45eb
Normal-740.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-740.png,Normal/images/Normal-740.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-740.png,Normal/masks/Normal-740.png,True,False,299,299,L,114.01,f13ea47277853943ffad27be7a3924ca
Normal-7400.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7400.png,Normal/images/Normal-7400.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7400.png,Normal/masks/Normal-7400.png,True,False,299,299,L,109.65,7db6623c89ed42a61e92740c2597ba70
Normal-7401.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7401.png,Normal/images/Normal-7401.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7401.png,Normal/masks/Normal-7401.png,True,False,299,299,L,146.22,94bafb853f74ea80a1a6dcaeabb2daac
Normal-7402.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7402.png,Normal/images/Normal-7402.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7402.png,Normal/masks/Normal-7402.png,True,False,299,299,L,109.87,cac3a75df3c0ac8eba7fd67b0e353cd5
Normal-7403.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7403.png,Normal/images/Normal-7403.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7403.png,Normal/masks/Normal-7403.png,True,False,299,299,L,150.03,90a439f938df5595c31103ddb9c5f6ca
Normal-7404.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7404.png,Normal/images/Normal-7404.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7404.png,Normal/masks/Normal-7404.png,True,False,299,299,L,132.78,9153e6401fee38a8880c607498075296
Normal-7405.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7405.png,Normal/images/Normal-7405.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7405.png,Normal/masks/Normal-7405.png,True,False,299,299,L,131.39,c7fc255c8b4410e53bc4e6393c7c7aeb
Normal-7406.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7406.png,Normal/images/Normal-7406.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7406.png,Normal/masks/Normal-7406.png,True,False,299,299,L,129.6,9ee52a67c49bd134c8ebf96a6fdb2d5a
Normal-7407.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7407.png,Normal/images/Normal-7407.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7407.png,Normal/masks/Normal-7407.png,True,False,299,299,L,127.16,209f7ce2527cf8b01037d0ad4a181af9
Normal-7408.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7408.png,Normal/images/Normal-7408.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7408.png,Normal/masks/Normal-7408.png,True,False,299,299,L,129.5,f7346c874188ac7f0e3041dce3989a0d
Normal-7409.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7409.png,Normal/images/Normal-7409.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7409.png,Normal/masks/Normal-7409.png,True,False,299,299,L,171.67,97118e1233180e8a7672127c25927162
Normal-741.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-741.png,Normal/images/Normal-741.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-741.png,Normal/masks/Normal-741.png,True,False,299,299,L,108.55,f8753d1326603d3e76e310ae3fbd38db
Normal-7410.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7410.png,Normal/images/Normal-7410.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7410.png,Normal/masks/Normal-7410.png,True,False,299,299,L,143.94,033aafb076065d7985a4e96dfbd5c621
Normal-7411.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7411.png,Normal/images/Normal-7411.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7411.png,Normal/masks/Normal-7411.png,True,False,299,299,L,133.33,f701a50c91d58128ece5a4c6e6d132b2
Normal-7412.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7412.png,Normal/images/Normal-7412.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7412.png,Normal/masks/Normal-7412.png,True,False,299,299,L,126.23,29057342c7cb5d9bf819cf59eee93784
Normal-7413.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7413.png,Normal/images/Normal-7413.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7413.png,Normal/masks/Normal-7413.png,True,False,299,299,L,103.05,abe92f09cfa435082c60b7d145f84c6d
Normal-7414.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7414.png,Normal/images/Normal-7414.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7414.png,Normal/masks/Normal-7414.png,True,False,299,299,L,177.24,8d9229759f143e04b2ff0362f5e0d998
Normal-7415.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7415.png,Normal/images/Normal-7415.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7415.png,Normal/masks/Normal-7415.png,True,False,299,299,L,103.44,1bf68104732251eebf43cd5259d9f5af
Normal-7416.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7416.png,Normal/images/Normal-7416.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7416.png,Normal/masks/Normal-7416.png,True,False,299,299,L,126.36,c144dfc8bbfa06fb2fdefcdc69e0cddc
Normal-7417.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7417.png,Normal/images/Normal-7417.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7417.png,Normal/masks/Normal-7417.png,True,False,299,299,L,173.52,602aa1cf2d8e917a6c37b5e996b4563d
Normal-7418.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7418.png,Normal/images/Normal-7418.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7418.png,Normal/masks/Normal-7418.png,True,False,299,299,L,118.62,26f5a2c206d84464bf37971c7fbbe384
Normal-7419.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7419.png,Normal/images/Normal-7419.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7419.png,Normal/masks/Normal-7419.png,True,False,299,299,L,162.85,ead9f547f703538337fd6fcb894182a8
Normal-742.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-742.png,Normal/images/Normal-742.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-742.png,Normal/masks/Normal-742.png,True,False,299,299,L,130.02,6af6f455bd523ecbd4f2dfe62753f864
Normal-7420.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7420.png,Normal/images/Normal-7420.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7420.png,Normal/masks/Normal-7420.png,True,False,299,299,L,97.96,328cd082937c43eabb9b8d193089a1d7
Normal-7421.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7421.png,Normal/images/Normal-7421.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7421.png,Normal/masks/Normal-7421.png,True,False,299,299,L,138.31,6ddc1b059d707733837e67d62fbbdd7a
Normal-7422.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7422.png,Normal/images/Normal-7422.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7422.png,Normal/masks/Normal-7422.png,True,False,299,299,L,135.78,beb102f542c3a9d42b79bb4285a48519
Normal-7423.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7423.png,Normal/images/Normal-7423.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7423.png,Normal/masks/Normal-7423.png,True,False,299,299,L,118.3,c36599544022a743d52f8fb13bdd9cea
Normal-7424.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7424.png,Normal/images/Normal-7424.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7424.png,Normal/masks/Normal-7424.png,True,False,299,299,L,150.19,a735a7aa18eb7762c6676579b366b045
Normal-7425.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7425.png,Normal/images/Normal-7425.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7425.png,Normal/masks/Normal-7425.png,True,False,299,299,L,130.26,dd2bd7e32b2bc1505e25afbfda299a15
Normal-7426.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7426.png,Normal/images/Normal-7426.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7426.png,Normal/masks/Normal-7426.png,True,False,299,299,L,131.33,6567de59f2b042fb0d9b09fbdab8311a
Normal-7427.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7427.png,Normal/images/Normal-7427.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7427.png,Normal/masks/Normal-7427.png,True,False,299,299,L,132.23,0dffa73338cb47c48bc075a27524ae82
Normal-7428.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7428.png,Normal/images/Normal-7428.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7428.png,Normal/masks/Normal-7428.png,True,False,299,299,L,93.82,7dd70fff0a43fe74e88a95834579eb94
Normal-7429.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7429.png,Normal/images/Normal-7429.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7429.png,Normal/masks/Normal-7429.png,True,False,299,299,L,144.11,c44fb3fa7c11eaded2ada577781f2204
Normal-743.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-743.png,Normal/images/Normal-743.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-743.png,Normal/masks/Normal-743.png,True,False,299,299,L,130.58,0b0414be016a3434f9d2c2f6694a49d2
Normal-7430.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7430.png,Normal/images/Normal-7430.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7430.png,Normal/masks/Normal-7430.png,True,False,299,299,L,127.22,27f1ae6096ef76c4b0a041538dbb8470
Normal-7431.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7431.png,Normal/images/Normal-7431.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7431.png,Normal/masks/Normal-7431.png,True,False,299,299,L,109.93,42433ce82eb70f678f2f268794ce40de
Normal-7432.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7432.png,Normal/images/Normal-7432.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7432.png,Normal/masks/Normal-7432.png,True,False,299,299,L,135.32,1558b83269325ce72a210a74ec667a63
Normal-7433.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7433.png,Normal/images/Normal-7433.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7433.png,Normal/masks/Normal-7433.png,True,False,299,299,L,178.4,257a5892136d888a0c82321f993834a4
Normal-7434.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7434.png,Normal/images/Normal-7434.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7434.png,Normal/masks/Normal-7434.png,True,False,299,299,L,152.72,f64b197c99acc9310e8399052b2f3a1a
Normal-7435.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7435.png,Normal/images/Normal-7435.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7435.png,Normal/masks/Normal-7435.png,True,False,299,299,L,109.79,91eed3b2cdb1a0b62f959d4f2c060547
Normal-7436.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7436.png,Normal/images/Normal-7436.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7436.png,Normal/masks/Normal-7436.png,True,False,299,299,L,124.79,3d6ecf809c2f0094ee86423b0d842930
Normal-7437.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7437.png,Normal/images/Normal-7437.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7437.png,Normal/masks/Normal-7437.png,True,False,299,299,L,100.68,3cd220001253a34d31fab2e9151fabb9
Normal-7438.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7438.png,Normal/images/Normal-7438.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7438.png,Normal/masks/Normal-7438.png,True,False,299,299,L,112.63,e7fd0a1098cbcfa37a9ad1795491fe04
Normal-7439.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7439.png,Normal/images/Normal-7439.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7439.png,Normal/masks/Normal-7439.png,True,False,299,299,L,115.94,83af6a25ad37f1a4cdbe94b47669204a
Normal-744.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-744.png,Normal/images/Normal-744.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-744.png,Normal/masks/Normal-744.png,True,False,299,299,L,105.16,d47f10c65ecf6a9b490b01df62c6018e
Normal-7440.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7440.png,Normal/images/Normal-7440.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7440.png,Normal/masks/Normal-7440.png,True,False,299,299,L,125.4,1e3a360ea02462c12dbfd04186f7fe18
Normal-7441.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7441.png,Normal/images/Normal-7441.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7441.png,Normal/masks/Normal-7441.png,True,False,299,299,L,128.61,9850e16860ce29ee36447b861b432b9b
Normal-7442.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7442.png,Normal/images/Normal-7442.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7442.png,Normal/masks/Normal-7442.png,True,False,299,299,L,176.37,e62b9408a923e2d37b4377d8c83431ba
Normal-7443.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7443.png,Normal/images/Normal-7443.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7443.png,Normal/masks/Normal-7443.png,True,False,299,299,L,138.47,e94053d0db243effad04ea9a20fccc02
Normal-7444.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7444.png,Normal/images/Normal-7444.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7444.png,Normal/masks/Normal-7444.png,True,False,299,299,L,153.08,233e934c6e18bdc024da0a90d0eb169d
Normal-7445.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7445.png,Normal/images/Normal-7445.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7445.png,Normal/masks/Normal-7445.png,True,False,299,299,L,128.44,0ada65f061e6faa881edbce3ae294090
Normal-7446.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7446.png,Normal/images/Normal-7446.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7446.png,Normal/masks/Normal-7446.png,True,False,299,299,L,154.42,57502d12977e1b7285711b01339e18b9
Normal-7447.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7447.png,Normal/images/Normal-7447.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7447.png,Normal/masks/Normal-7447.png,True,False,299,299,L,98.43,31c4d8aef320f8b473137b5769cd33d9
Normal-7448.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7448.png,Normal/images/Normal-7448.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7448.png,Normal/masks/Normal-7448.png,True,False,299,299,L,122.43,f4f0a6e677d30b17439f9e9e3a49967f
Normal-7449.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7449.png,Normal/images/Normal-7449.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7449.png,Normal/masks/Normal-7449.png,True,False,299,299,L,99.82,7690f7ba48d797e4a442e5d5f0682b2e
Normal-745.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-745.png,Normal/images/Normal-745.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-745.png,Normal/masks/Normal-745.png,True,False,299,299,L,111.32,08ec87e82e69783157cbf19f06c1bd7d
Normal-7450.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7450.png,Normal/images/Normal-7450.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7450.png,Normal/masks/Normal-7450.png,True,False,299,299,L,122.78,3a44aa8df743036b893e13cd6ad20efb
Normal-7451.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7451.png,Normal/images/Normal-7451.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7451.png,Normal/masks/Normal-7451.png,True,False,299,299,L,147.74,f96cff2e46cfb9a819e1e049bcc3aa05
Normal-7452.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7452.png,Normal/images/Normal-7452.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7452.png,Normal/masks/Normal-7452.png,True,False,299,299,L,115.37,ac2f0bc20260634fde98e590d8775b17
Normal-7453.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7453.png,Normal/images/Normal-7453.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7453.png,Normal/masks/Normal-7453.png,True,False,299,299,L,125.52,2ab51ac507560f3dd80b0c72f01f6bac
Normal-7454.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7454.png,Normal/images/Normal-7454.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7454.png,Normal/masks/Normal-7454.png,True,False,299,299,L,119.34,95b3cfdc2c2f4bd9f09d2d659a7de80e
Normal-7455.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7455.png,Normal/images/Normal-7455.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7455.png,Normal/masks/Normal-7455.png,True,False,299,299,L,136.39,551f35d566d2a18f11b6b83c749b07ad
Normal-7456.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7456.png,Normal/images/Normal-7456.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7456.png,Normal/masks/Normal-7456.png,True,False,299,299,L,145.43,0227e211cdce9d4a0e24ee0c073fcc21
Normal-7457.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7457.png,Normal/images/Normal-7457.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7457.png,Normal/masks/Normal-7457.png,True,False,299,299,L,171.58,0f61e6f26afd43750ac1bc19f3a88cbe
Normal-7458.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7458.png,Normal/images/Normal-7458.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7458.png,Normal/masks/Normal-7458.png,True,False,299,299,L,87.65,ed25b8eb607f80a5b1b12f253ca4ef0c
Normal-7459.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7459.png,Normal/images/Normal-7459.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7459.png,Normal/masks/Normal-7459.png,True,False,299,299,L,148.33,e4dd4a02c512eb7e82115b2f34eaadce
Normal-746.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-746.png,Normal/images/Normal-746.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-746.png,Normal/masks/Normal-746.png,True,False,299,299,L,133.68,fef86c79f206b408bfb557607f8d275a
Normal-7460.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7460.png,Normal/images/Normal-7460.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7460.png,Normal/masks/Normal-7460.png,True,False,299,299,L,107.19,c8b9954460e468e1dd9558f9aa8ee78b
Normal-7461.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7461.png,Normal/images/Normal-7461.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7461.png,Normal/masks/Normal-7461.png,True,False,299,299,L,113.22,bc5693cfe7fc3d1e97f6f81fa16b3ad9
Normal-7462.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7462.png,Normal/images/Normal-7462.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7462.png,Normal/masks/Normal-7462.png,True,False,299,299,L,162.93,7724b0013478433caab311b56df01718
Normal-7463.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7463.png,Normal/images/Normal-7463.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7463.png,Normal/masks/Normal-7463.png,True,False,299,299,L,109.32,5c4467ed4b3b776dd8e91594afd7dda3
Normal-7464.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7464.png,Normal/images/Normal-7464.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7464.png,Normal/masks/Normal-7464.png,True,False,299,299,L,140.93,e2d6238818f7401a808923c7605c8ebc
Normal-7465.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7465.png,Normal/images/Normal-7465.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7465.png,Normal/masks/Normal-7465.png,True,False,299,299,L,170.85,2987ad65f639d9fa54f967b90914f480
Normal-7466.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7466.png,Normal/images/Normal-7466.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7466.png,Normal/masks/Normal-7466.png,True,False,299,299,L,121.72,dcc45d1a1c7faf20879d0b7d397c6773
Normal-7467.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7467.png,Normal/images/Normal-7467.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7467.png,Normal/masks/Normal-7467.png,True,False,299,299,L,133.49,8d91618b113a0261a5b460ab76cda213
Normal-7468.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7468.png,Normal/images/Normal-7468.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7468.png,Normal/masks/Normal-7468.png,True,False,299,299,L,127.68,fe37713f3ae2ee0aeb562864fa39c7ae
Normal-7469.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7469.png,Normal/images/Normal-7469.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7469.png,Normal/masks/Normal-7469.png,True,False,299,299,L,112.08,4ac41c88148ab7aa75450d3087083ea4
Normal-747.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-747.png,Normal/images/Normal-747.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-747.png,Normal/masks/Normal-747.png,True,False,299,299,L,119.87,9e668d7e47cc6f3def60face71d6b5c6
Normal-7470.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7470.png,Normal/images/Normal-7470.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7470.png,Normal/masks/Normal-7470.png,True,False,299,299,L,118.34,426bba939fe040b4e5741c122179a44d
Normal-7471.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7471.png,Normal/images/Normal-7471.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7471.png,Normal/masks/Normal-7471.png,True,False,299,299,L,126.8,8bcf2a11a06b497646557257bb3ff4c7
Normal-7472.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7472.png,Normal/images/Normal-7472.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7472.png,Normal/masks/Normal-7472.png,True,False,299,299,L,109.34,db7a09b6de3f2d0650ee47eda0b47576
Normal-7473.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7473.png,Normal/images/Normal-7473.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7473.png,Normal/masks/Normal-7473.png,True,False,299,299,L,145.06,09ff4655c561d0e0cf30991d0bd68ad8
Normal-7474.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7474.png,Normal/images/Normal-7474.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7474.png,Normal/masks/Normal-7474.png,True,False,299,299,L,165.68,e73abb3469b6ffbf8277122f7712da79
Normal-7475.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7475.png,Normal/images/Normal-7475.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7475.png,Normal/masks/Normal-7475.png,True,False,299,299,L,157.28,750a3ff45adb5482d88f17c93e5cff00
Normal-7476.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7476.png,Normal/images/Normal-7476.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7476.png,Normal/masks/Normal-7476.png,True,False,299,299,L,119.88,d5160e45fe6b5049fb159bfbbcb03eb5
Normal-7477.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7477.png,Normal/images/Normal-7477.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7477.png,Normal/masks/Normal-7477.png,True,False,299,299,L,133.25,c94dcc6b04246bca035e6263db393b19
Normal-7478.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7478.png,Normal/images/Normal-7478.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7478.png,Normal/masks/Normal-7478.png,True,False,299,299,L,152.77,ac18f03532d0d8cdaa8ef4efa6cf6892
Normal-7479.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7479.png,Normal/images/Normal-7479.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7479.png,Normal/masks/Normal-7479.png,True,False,299,299,L,134.75,7ceb3e17dd3288aa1880aa17189fb78f
Normal-748.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-748.png,Normal/images/Normal-748.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-748.png,Normal/masks/Normal-748.png,True,False,299,299,L,129.07,3af7bdea126e48d492db760c9f0b2fc7
Normal-7480.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7480.png,Normal/images/Normal-7480.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7480.png,Normal/masks/Normal-7480.png,True,False,299,299,L,158.85,c3e0cb8dce8342b06944924b8c1cc77c
Normal-7481.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7481.png,Normal/images/Normal-7481.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7481.png,Normal/masks/Normal-7481.png,True,False,299,299,L,109.3,f27c0ec2c70cdeba0178271cf2583885
Normal-7482.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7482.png,Normal/images/Normal-7482.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7482.png,Normal/masks/Normal-7482.png,True,False,299,299,L,124.86,9d85b97634099536847e87291ef49a09
Normal-7483.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7483.png,Normal/images/Normal-7483.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7483.png,Normal/masks/Normal-7483.png,True,False,299,299,L,131.02,af686d919df0d66e605748d0560f2352
Normal-7484.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7484.png,Normal/images/Normal-7484.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7484.png,Normal/masks/Normal-7484.png,True,False,299,299,L,118.55,f0a8bd92eaeb9c131fab83e4aec1af3f
Normal-7485.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7485.png,Normal/images/Normal-7485.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7485.png,Normal/masks/Normal-7485.png,True,False,299,299,L,120.1,6d7c23b2c1d76a7ec28341dbc82e9b52
Normal-7486.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7486.png,Normal/images/Normal-7486.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7486.png,Normal/masks/Normal-7486.png,True,False,299,299,L,111.24,6c3ae29b7653366a705e8f5e0dd8fd5e
Normal-7487.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7487.png,Normal/images/Normal-7487.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7487.png,Normal/masks/Normal-7487.png,True,False,299,299,L,119.33,1ae3deffa824a513868056b575ea8cb7
Normal-7488.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7488.png,Normal/images/Normal-7488.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7488.png,Normal/masks/Normal-7488.png,True,False,299,299,L,180.73,b9a3e835c85304a6b88fb28e3e2cad75
Normal-7489.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7489.png,Normal/images/Normal-7489.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7489.png,Normal/masks/Normal-7489.png,True,False,299,299,L,148.43,c1a8bcea9d88af62a274d16ee578d40b
Normal-749.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-749.png,Normal/images/Normal-749.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-749.png,Normal/masks/Normal-749.png,True,False,299,299,L,102.22,303da3b8c20e227e37be5bdc67a1465a
Normal-7490.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7490.png,Normal/images/Normal-7490.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7490.png,Normal/masks/Normal-7490.png,True,False,299,299,L,102.55,f96550d56684624aabfdd53bea453580
Normal-7491.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7491.png,Normal/images/Normal-7491.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7491.png,Normal/masks/Normal-7491.png,True,False,299,299,L,65.41,13b9045e3f98784aa1bdf1ae166ca291
Normal-7492.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7492.png,Normal/images/Normal-7492.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7492.png,Normal/masks/Normal-7492.png,True,False,299,299,L,63.34,cb9810f7d9cef2e92a30d9bc19b76c51
Normal-7493.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7493.png,Normal/images/Normal-7493.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7493.png,Normal/masks/Normal-7493.png,True,False,299,299,L,147.45,b3e5a3d16e431ac6eaa42df5a39efceb
Normal-7494.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7494.png,Normal/images/Normal-7494.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7494.png,Normal/masks/Normal-7494.png,True,False,299,299,L,128.7,febcb572825bf8fb8ef35d365f503b0f
Normal-7495.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7495.png,Normal/images/Normal-7495.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7495.png,Normal/masks/Normal-7495.png,True,False,299,299,L,171.74,78ca6b80fa60ea5359f9eb7cedf5abc1
Normal-7496.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7496.png,Normal/images/Normal-7496.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7496.png,Normal/masks/Normal-7496.png,True,False,299,299,L,122.43,88de7225178fed815593fff19515f85f
Normal-7497.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7497.png,Normal/images/Normal-7497.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7497.png,Normal/masks/Normal-7497.png,True,False,299,299,L,131.93,6b168b99c8779e419d881991e2dde34a
Normal-7498.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7498.png,Normal/images/Normal-7498.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7498.png,Normal/masks/Normal-7498.png,True,False,299,299,L,130.74,b7662ad69ff2527da427fd2027f62cd5
Normal-7499.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7499.png,Normal/images/Normal-7499.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7499.png,Normal/masks/Normal-7499.png,True,False,299,299,L,156.25,0001babf4b418bc80722c4ca94b3e03d
Normal-75.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-75.png,Normal/images/Normal-75.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-75.png,Normal/masks/Normal-75.png,True,False,299,299,L,148.89,ee7752cce599a61379a9b7d38881db03
Normal-750.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-750.png,Normal/images/Normal-750.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-750.png,Normal/masks/Normal-750.png,True,False,299,299,L,108.69,88f1b5dc088538631f47529b4ca7a690
Normal-7500.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7500.png,Normal/images/Normal-7500.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7500.png,Normal/masks/Normal-7500.png,True,False,299,299,L,150.35,636221b7bfc539cb7c3992c153f6a7b5
Normal-7501.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7501.png,Normal/images/Normal-7501.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7501.png,Normal/masks/Normal-7501.png,True,False,299,299,L,124.51,190650abcb3e5d140014ce5c0cd42b7f
Normal-7502.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7502.png,Normal/images/Normal-7502.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7502.png,Normal/masks/Normal-7502.png,True,False,299,299,L,112.54,6e9fac46d46b8bb90bfc4ef81fdd079d
Normal-7503.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7503.png,Normal/images/Normal-7503.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7503.png,Normal/masks/Normal-7503.png,True,False,299,299,L,147.37,c20d7bf3da280560636507dbd70a8059
Normal-7504.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7504.png,Normal/images/Normal-7504.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7504.png,Normal/masks/Normal-7504.png,True,False,299,299,L,107.99,c5ae58d63b8133a910fb39593502f7ec
Normal-7505.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7505.png,Normal/images/Normal-7505.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7505.png,Normal/masks/Normal-7505.png,True,False,299,299,L,125.99,6a1d654e7f3ae54590a13ae41eaa974a
Normal-7506.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7506.png,Normal/images/Normal-7506.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7506.png,Normal/masks/Normal-7506.png,True,False,299,299,L,129.11,4984d61c135b2b8183b8d507f9edc7ec
Normal-7507.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7507.png,Normal/images/Normal-7507.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7507.png,Normal/masks/Normal-7507.png,True,False,299,299,L,143.43,81c9481c4a035204f500aca5038f799f
Normal-7508.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7508.png,Normal/images/Normal-7508.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7508.png,Normal/masks/Normal-7508.png,True,False,299,299,L,128.06,cc788f3c45988c63383a9772cd2d1de4
Normal-7509.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7509.png,Normal/images/Normal-7509.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7509.png,Normal/masks/Normal-7509.png,True,False,299,299,L,127.39,fbc2274ab2c73cef9d135a366766daba
Normal-751.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-751.png,Normal/images/Normal-751.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-751.png,Normal/masks/Normal-751.png,True,False,299,299,L,128.01,c492957adf5ba4528799736d66885307
Normal-7510.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7510.png,Normal/images/Normal-7510.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7510.png,Normal/masks/Normal-7510.png,True,False,299,299,L,161.87,055f75c5f224ce2d0faebca032a37fb2
Normal-7511.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7511.png,Normal/images/Normal-7511.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7511.png,Normal/masks/Normal-7511.png,True,False,299,299,L,117.59,2934a94704fdc282fe4c16a79a645d77
Normal-7512.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7512.png,Normal/images/Normal-7512.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7512.png,Normal/masks/Normal-7512.png,True,False,299,299,L,180.75,826133c5a7479aa81c33847c06fbd0d8
Normal-7513.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7513.png,Normal/images/Normal-7513.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7513.png,Normal/masks/Normal-7513.png,True,False,299,299,L,177.08,63c4848f2688681ce471501be1777186
Normal-7514.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7514.png,Normal/images/Normal-7514.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7514.png,Normal/masks/Normal-7514.png,True,False,299,299,L,128.92,2057d58a291aa7b7ab5a53d5397d49e8
Normal-7515.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7515.png,Normal/images/Normal-7515.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7515.png,Normal/masks/Normal-7515.png,True,False,299,299,L,142.43,f7aeb8a6d62bf6415ed9437c4b4e6adb
Normal-7516.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7516.png,Normal/images/Normal-7516.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7516.png,Normal/masks/Normal-7516.png,True,False,299,299,L,144.1,1e602c93a025958d34a2f7292caea992
Normal-7517.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7517.png,Normal/images/Normal-7517.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7517.png,Normal/masks/Normal-7517.png,True,False,299,299,L,155.79,5106859021fe6a17d18f6db85de8bab9
Normal-7518.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7518.png,Normal/images/Normal-7518.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7518.png,Normal/masks/Normal-7518.png,True,False,299,299,L,154.14,60349fbca4e22d224a4d2d6e6e6a7fe2
Normal-7519.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7519.png,Normal/images/Normal-7519.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7519.png,Normal/masks/Normal-7519.png,True,False,299,299,L,167.28,cbd20f4fc996ab20da060ed6398442f5
Normal-752.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-752.png,Normal/images/Normal-752.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-752.png,Normal/masks/Normal-752.png,True,False,299,299,L,124.14,b52693531c88598527a7890f790c26aa
Normal-7520.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7520.png,Normal/images/Normal-7520.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7520.png,Normal/masks/Normal-7520.png,True,False,299,299,L,110.42,87c7d637554ba14ad8d82b2262a894bf
Normal-7521.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7521.png,Normal/images/Normal-7521.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7521.png,Normal/masks/Normal-7521.png,True,False,299,299,L,155.41,0b043e8ee3c0e03f60a889caf8502066
Normal-7522.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7522.png,Normal/images/Normal-7522.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7522.png,Normal/masks/Normal-7522.png,True,False,299,299,L,159.2,e580a0a7751d72c25de1e7c25e490eb5
Normal-7523.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7523.png,Normal/images/Normal-7523.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7523.png,Normal/masks/Normal-7523.png,True,False,299,299,L,147.91,a9e528115a47fe36250daa9ec1f16271
Normal-7524.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7524.png,Normal/images/Normal-7524.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7524.png,Normal/masks/Normal-7524.png,True,False,299,299,L,105.51,8f2fc809c17929a8f5ed51f1358a394d
Normal-7525.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7525.png,Normal/images/Normal-7525.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7525.png,Normal/masks/Normal-7525.png,True,False,299,299,L,112.38,6ae4ab0952602ad02d716765e2364e52
Normal-7526.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7526.png,Normal/images/Normal-7526.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7526.png,Normal/masks/Normal-7526.png,True,False,299,299,L,159.32,ab7be9eb6b1e433bcbd87b2ea8e69e7b
Normal-7527.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7527.png,Normal/images/Normal-7527.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7527.png,Normal/masks/Normal-7527.png,True,False,299,299,L,114.14,d7ba31360e0f0645d87fc7d6f62ee141
Normal-7528.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7528.png,Normal/images/Normal-7528.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7528.png,Normal/masks/Normal-7528.png,True,False,299,299,L,164.04,3f4af8aad5dd253b0e197849a2b1b2e1
Normal-7529.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7529.png,Normal/images/Normal-7529.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7529.png,Normal/masks/Normal-7529.png,True,False,299,299,L,189.71,efa552f188e500228cdf14b1675c90b0
Normal-753.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-753.png,Normal/images/Normal-753.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-753.png,Normal/masks/Normal-753.png,True,False,299,299,L,127.69,a8c98a96df309d23e5f70a06492eabba
Normal-7530.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7530.png,Normal/images/Normal-7530.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7530.png,Normal/masks/Normal-7530.png,True,False,299,299,L,107.32,653933ad361fcca5f2461cf34b006614
Normal-7531.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7531.png,Normal/images/Normal-7531.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7531.png,Normal/masks/Normal-7531.png,True,False,299,299,L,144.99,5b956ae0a73cf1c32605de02866e129c
Normal-7532.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7532.png,Normal/images/Normal-7532.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7532.png,Normal/masks/Normal-7532.png,True,False,299,299,L,161.39,b736ddc792b76d407ddbc1bed1fe71a0
Normal-7533.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7533.png,Normal/images/Normal-7533.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7533.png,Normal/masks/Normal-7533.png,True,False,299,299,L,120.62,38f22d60ff63b2cfb5ecbe7ecb05c0d8
Normal-7534.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7534.png,Normal/images/Normal-7534.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7534.png,Normal/masks/Normal-7534.png,True,False,299,299,L,129.87,53eda51c0c7cdb2f758cf9fc1d930201
Normal-7535.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7535.png,Normal/images/Normal-7535.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7535.png,Normal/masks/Normal-7535.png,True,False,299,299,L,161.13,8d48a59a5278547d53f040a6a9e8c5a9
Normal-7536.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7536.png,Normal/images/Normal-7536.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7536.png,Normal/masks/Normal-7536.png,True,False,299,299,L,150.69,53b455d2e10c41ebe545962ea7ef87a6
Normal-7537.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7537.png,Normal/images/Normal-7537.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7537.png,Normal/masks/Normal-7537.png,True,False,299,299,L,114.52,3ab9c2b5a672421734d534ebb43e2b67
Normal-7538.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7538.png,Normal/images/Normal-7538.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7538.png,Normal/masks/Normal-7538.png,True,False,299,299,L,128.79,1bb768734955d7bcfbdd47cf74842b4c
Normal-7539.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7539.png,Normal/images/Normal-7539.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7539.png,Normal/masks/Normal-7539.png,True,False,299,299,L,113.95,d7dfbf0e94f3c669845893f592c3e12c
Normal-754.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-754.png,Normal/images/Normal-754.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-754.png,Normal/masks/Normal-754.png,True,False,299,299,L,138.83,e4a9374439ab0238d8482dc198dd1d14
Normal-7540.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7540.png,Normal/images/Normal-7540.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7540.png,Normal/masks/Normal-7540.png,True,False,299,299,L,108.88,c0a54a6de82f68faa723d99d37e04086
Normal-7541.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7541.png,Normal/images/Normal-7541.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7541.png,Normal/masks/Normal-7541.png,True,False,299,299,L,133.17,193df663333034015a0afbccae65ddf4
Normal-7542.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7542.png,Normal/images/Normal-7542.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7542.png,Normal/masks/Normal-7542.png,True,False,299,299,L,167.74,91b003273a47e50b1206545e24aed537
Normal-7543.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7543.png,Normal/images/Normal-7543.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7543.png,Normal/masks/Normal-7543.png,True,False,299,299,L,107.43,0dc4f5dea911a35a7995108b84445c47
Normal-7544.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7544.png,Normal/images/Normal-7544.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7544.png,Normal/masks/Normal-7544.png,True,False,299,299,L,115.17,629bd98526132201a695c6ad74aa48e2
Normal-7545.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7545.png,Normal/images/Normal-7545.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7545.png,Normal/masks/Normal-7545.png,True,False,299,299,L,153.96,bd13de3b5878066f2c86bf82b65d7eee
Normal-7546.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7546.png,Normal/images/Normal-7546.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7546.png,Normal/masks/Normal-7546.png,True,False,299,299,L,95.9,8e7212bffd777e26a571fe4d1479b4cc
Normal-7547.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7547.png,Normal/images/Normal-7547.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7547.png,Normal/masks/Normal-7547.png,True,False,299,299,L,125.92,9d90faee3144f0be51deb56dc3d9a8c6
Normal-7548.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7548.png,Normal/images/Normal-7548.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7548.png,Normal/masks/Normal-7548.png,True,False,299,299,L,137.76,05b13b85e673fdb37ffc324c5d1c0e73
Normal-7549.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7549.png,Normal/images/Normal-7549.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7549.png,Normal/masks/Normal-7549.png,True,False,299,299,L,106.38,4dd798815620edbf5343fb3dfc09660e
Normal-755.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-755.png,Normal/images/Normal-755.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-755.png,Normal/masks/Normal-755.png,True,False,299,299,L,123.46,89d9f7db1271990ea0547ceae4b1be75
Normal-7550.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7550.png,Normal/images/Normal-7550.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7550.png,Normal/masks/Normal-7550.png,True,False,299,299,L,117.63,294161fe2b750047773695a41c62b0bb
Normal-7551.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7551.png,Normal/images/Normal-7551.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7551.png,Normal/masks/Normal-7551.png,True,False,299,299,L,175.57,76060c12f1ca8c73f9326b47cb12e7fc
Normal-7552.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7552.png,Normal/images/Normal-7552.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7552.png,Normal/masks/Normal-7552.png,True,False,299,299,L,123.85,e9ff388662f02a16d19b0f8f06881731
Normal-7553.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7553.png,Normal/images/Normal-7553.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7553.png,Normal/masks/Normal-7553.png,True,False,299,299,L,104.96,9ef61abd2da3a09a371877fcaa615b06
Normal-7554.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7554.png,Normal/images/Normal-7554.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7554.png,Normal/masks/Normal-7554.png,True,False,299,299,L,102.71,858a99abec79d352f72d8ef3ee746d95
Normal-7555.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7555.png,Normal/images/Normal-7555.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7555.png,Normal/masks/Normal-7555.png,True,False,299,299,L,81.06,03909cb7490641caa1b937ecfaae41f6
Normal-7556.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7556.png,Normal/images/Normal-7556.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7556.png,Normal/masks/Normal-7556.png,True,False,299,299,L,154.08,9201cea3bdc5e6b41d9b4acea02fc506
Normal-7557.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7557.png,Normal/images/Normal-7557.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7557.png,Normal/masks/Normal-7557.png,True,False,299,299,L,111.84,1cef0ad7edf927e5050715ff22e56169
Normal-7558.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7558.png,Normal/images/Normal-7558.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7558.png,Normal/masks/Normal-7558.png,True,False,299,299,L,114.66,e478e8fc4821bdeb26951f40940ed402
Normal-7559.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7559.png,Normal/images/Normal-7559.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7559.png,Normal/masks/Normal-7559.png,True,False,299,299,L,135.62,4e6fe0a2b1bb539639357924813e12b4
Normal-756.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-756.png,Normal/images/Normal-756.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-756.png,Normal/masks/Normal-756.png,True,False,299,299,L,110.09,443571f56760ae392b2a8a638dca6ce9
Normal-7560.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7560.png,Normal/images/Normal-7560.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7560.png,Normal/masks/Normal-7560.png,True,False,299,299,L,110.64,fcb222b4454aaad570c30d1373f6c6be
Normal-7561.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7561.png,Normal/images/Normal-7561.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7561.png,Normal/masks/Normal-7561.png,True,False,299,299,L,105.07,8c14411aac98981d03fc7db3e4f0c612
Normal-7562.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7562.png,Normal/images/Normal-7562.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7562.png,Normal/masks/Normal-7562.png,True,False,299,299,L,118.18,12bba3103be8500928080088aa19ebd7
Normal-7563.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7563.png,Normal/images/Normal-7563.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7563.png,Normal/masks/Normal-7563.png,True,False,299,299,L,117.05,f39dfc2e5616c5848c59578b0f9b83c5
Normal-7564.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7564.png,Normal/images/Normal-7564.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7564.png,Normal/masks/Normal-7564.png,True,False,299,299,L,159.38,c859725810b9c90038d8016b59f11819
Normal-7565.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7565.png,Normal/images/Normal-7565.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7565.png,Normal/masks/Normal-7565.png,True,False,299,299,L,124.86,d63c094bd2ea53cd701a28a5ad14eaf6
Normal-7566.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7566.png,Normal/images/Normal-7566.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7566.png,Normal/masks/Normal-7566.png,True,False,299,299,L,109.03,179e8d13c1a0102131f36f716e958905
Normal-7567.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7567.png,Normal/images/Normal-7567.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7567.png,Normal/masks/Normal-7567.png,True,False,299,299,L,73.76,0b5d7199317524fec6164b17f2142317
Normal-7568.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7568.png,Normal/images/Normal-7568.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7568.png,Normal/masks/Normal-7568.png,True,False,299,299,L,108.32,a41d65bb67c97256797ad58ee4b41a2a
Normal-7569.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7569.png,Normal/images/Normal-7569.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7569.png,Normal/masks/Normal-7569.png,True,False,299,299,L,120.62,03e607b5a15b0889108c0b7c430f7dad
Normal-757.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-757.png,Normal/images/Normal-757.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-757.png,Normal/masks/Normal-757.png,True,False,299,299,L,115.56,8dfae1323446f8901573a774dcbe9fea
Normal-7570.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7570.png,Normal/images/Normal-7570.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7570.png,Normal/masks/Normal-7570.png,True,False,299,299,L,148.56,6619fdeb6707525314d377e09138ae70
Normal-7571.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7571.png,Normal/images/Normal-7571.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7571.png,Normal/masks/Normal-7571.png,True,False,299,299,L,113.77,9223c2f9a7162fb02d2c65c4aa892fb7
Normal-7572.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7572.png,Normal/images/Normal-7572.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7572.png,Normal/masks/Normal-7572.png,True,False,299,299,L,123.57,3413f6fbcae9140db238634d5b45081c
Normal-7573.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7573.png,Normal/images/Normal-7573.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7573.png,Normal/masks/Normal-7573.png,True,False,299,299,L,119.07,fa6364e7991da4695f926b5b66d49950
Normal-7574.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7574.png,Normal/images/Normal-7574.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7574.png,Normal/masks/Normal-7574.png,True,False,299,299,L,137.78,9a9e2d662869615bd5cb70048354f769
Normal-7575.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7575.png,Normal/images/Normal-7575.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7575.png,Normal/masks/Normal-7575.png,True,False,299,299,L,132.37,ce3c12a3e0616c5fd615895196a26a38
Normal-7576.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7576.png,Normal/images/Normal-7576.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7576.png,Normal/masks/Normal-7576.png,True,False,299,299,L,153.92,8d82bfd3fe2c6f6134bf5732aea44b18
Normal-7577.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7577.png,Normal/images/Normal-7577.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7577.png,Normal/masks/Normal-7577.png,True,False,299,299,L,107.11,45bb94cab6feb76c408ea79ce54ae722
Normal-7578.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7578.png,Normal/images/Normal-7578.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7578.png,Normal/masks/Normal-7578.png,True,False,299,299,L,178.94,4ad5e3c182b62895f9145b6db20e3b65
Normal-7579.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7579.png,Normal/images/Normal-7579.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7579.png,Normal/masks/Normal-7579.png,True,False,299,299,L,112.2,05ce7c8a3d92bb1a607ccf2874897866
Normal-758.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-758.png,Normal/images/Normal-758.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-758.png,Normal/masks/Normal-758.png,True,False,299,299,L,107.92,28ca0cdd7ebf8b28f05e14b20d10678f
Normal-7580.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7580.png,Normal/images/Normal-7580.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7580.png,Normal/masks/Normal-7580.png,True,False,299,299,L,81.23,b52b5b2e967f4f5f08441957c01a07c9
Normal-7581.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7581.png,Normal/images/Normal-7581.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7581.png,Normal/masks/Normal-7581.png,True,False,299,299,L,122.42,64991031f82907400035d1f617081d1e
Normal-7582.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7582.png,Normal/images/Normal-7582.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7582.png,Normal/masks/Normal-7582.png,True,False,299,299,L,108.1,754f0c7df2a37a72bc2fb7a443b5429f
Normal-7583.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7583.png,Normal/images/Normal-7583.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7583.png,Normal/masks/Normal-7583.png,True,False,299,299,L,180.69,b2dcc3af2c2fe350098d6b3102ff935b
Normal-7584.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7584.png,Normal/images/Normal-7584.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7584.png,Normal/masks/Normal-7584.png,True,False,299,299,L,172.6,636ae29855187f3c4e06c2df0fc885ee
Normal-7585.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7585.png,Normal/images/Normal-7585.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7585.png,Normal/masks/Normal-7585.png,True,False,299,299,L,117.95,6242289c8a92a70084a9103c7bc52781
Normal-7586.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7586.png,Normal/images/Normal-7586.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7586.png,Normal/masks/Normal-7586.png,True,False,299,299,L,102.38,2153e111c6726d7e4be4a36486bbadfb
Normal-7587.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7587.png,Normal/images/Normal-7587.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7587.png,Normal/masks/Normal-7587.png,True,False,299,299,L,105.9,1d46b7c5d88e2708f198843e9d300bdb
Normal-7588.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7588.png,Normal/images/Normal-7588.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7588.png,Normal/masks/Normal-7588.png,True,False,299,299,L,126.23,41a4caa5485bedc9824dac926b8c39c7
Normal-7589.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7589.png,Normal/images/Normal-7589.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7589.png,Normal/masks/Normal-7589.png,True,False,299,299,L,161.65,5022cbe8712aaf687830f75831a2e685
Normal-759.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-759.png,Normal/images/Normal-759.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-759.png,Normal/masks/Normal-759.png,True,False,299,299,L,132.86,0ac67c647abcc22470323c626b86dd17
Normal-7590.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7590.png,Normal/images/Normal-7590.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7590.png,Normal/masks/Normal-7590.png,True,False,299,299,L,119.22,ed92ff22b065dfbba346c65bd0cf18ae
Normal-7591.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7591.png,Normal/images/Normal-7591.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7591.png,Normal/masks/Normal-7591.png,True,False,299,299,L,119.69,ed8766c0905b8b29daf7fc286c9f1c5f
Normal-7592.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7592.png,Normal/images/Normal-7592.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7592.png,Normal/masks/Normal-7592.png,True,False,299,299,L,87.57,67bb87c4707912e556eb8578c44550dd
Normal-7593.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7593.png,Normal/images/Normal-7593.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7593.png,Normal/masks/Normal-7593.png,True,False,299,299,L,123.39,89623d5d6139aaabe3e6399a1b64c22f
Normal-7594.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7594.png,Normal/images/Normal-7594.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7594.png,Normal/masks/Normal-7594.png,True,False,299,299,L,141.99,ce1ea6c5d3dd48402be07569010d321f
Normal-7595.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7595.png,Normal/images/Normal-7595.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7595.png,Normal/masks/Normal-7595.png,True,False,299,299,L,155.2,01ca5665359f6470bcbc7fa0200a03fa
Normal-7596.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7596.png,Normal/images/Normal-7596.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7596.png,Normal/masks/Normal-7596.png,True,False,299,299,L,121.98,f136c8bf4f5082635335ebcc0fc86fd4
Normal-7597.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7597.png,Normal/images/Normal-7597.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7597.png,Normal/masks/Normal-7597.png,True,False,299,299,L,121.07,b62bad09ef8cc0b6bbe79c17b4fd335b
Normal-7598.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7598.png,Normal/images/Normal-7598.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7598.png,Normal/masks/Normal-7598.png,True,False,299,299,L,111.63,eff483f5d164d230a09ad5a54d844aab
Normal-7599.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7599.png,Normal/images/Normal-7599.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7599.png,Normal/masks/Normal-7599.png,True,False,299,299,L,102.2,be2a2b86289286e45b6ac626ff48e17c
Normal-76.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-76.png,Normal/images/Normal-76.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-76.png,Normal/masks/Normal-76.png,True,False,299,299,L,118.15,db4c1293bad11ed9750a180ff62c0187
Normal-760.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-760.png,Normal/images/Normal-760.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-760.png,Normal/masks/Normal-760.png,True,False,299,299,L,128.61,21aceed54c12e80457d4c97e21e8e4e9
Normal-7600.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7600.png,Normal/images/Normal-7600.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7600.png,Normal/masks/Normal-7600.png,True,False,299,299,L,136.58,dbafcdb344c195269526dc0dd8506e65
Normal-7601.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7601.png,Normal/images/Normal-7601.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7601.png,Normal/masks/Normal-7601.png,True,False,299,299,L,80.32,da324bbb4cc699c107f580d4fcb01507
Normal-7602.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7602.png,Normal/images/Normal-7602.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7602.png,Normal/masks/Normal-7602.png,True,False,299,299,L,138.16,dc1db0d558c94eaa131f7b37be23d42c
Normal-7603.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7603.png,Normal/images/Normal-7603.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7603.png,Normal/masks/Normal-7603.png,True,False,299,299,L,163.69,9a5320fe1f8854223f4e278a64e600eb
Normal-7604.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7604.png,Normal/images/Normal-7604.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7604.png,Normal/masks/Normal-7604.png,True,False,299,299,L,98.46,5b7190e0a42249947951628b9e8298d7
Normal-7605.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7605.png,Normal/images/Normal-7605.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7605.png,Normal/masks/Normal-7605.png,True,False,299,299,L,165.23,fe6a03d8d32233f31d874b65ff4b4e22
Normal-7606.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7606.png,Normal/images/Normal-7606.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7606.png,Normal/masks/Normal-7606.png,True,False,299,299,L,116.13,ced8bbf6031435c4a8771cb5b2ded583
Normal-7607.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7607.png,Normal/images/Normal-7607.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7607.png,Normal/masks/Normal-7607.png,True,False,299,299,L,117.34,a5561de29c36f6f6c3509254ac7eb36f
Normal-7608.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7608.png,Normal/images/Normal-7608.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7608.png,Normal/masks/Normal-7608.png,True,False,299,299,L,145.24,fd638efceeab60ac426606c8fc65379f
Normal-7609.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7609.png,Normal/images/Normal-7609.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7609.png,Normal/masks/Normal-7609.png,True,False,299,299,L,152.97,74365909b0783edab98965868752b9be
Normal-761.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-761.png,Normal/images/Normal-761.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-761.png,Normal/masks/Normal-761.png,True,False,299,299,L,125.34,6d7c95130d7256695b003642f267ef7e
Normal-7610.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7610.png,Normal/images/Normal-7610.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7610.png,Normal/masks/Normal-7610.png,True,False,299,299,L,109.58,a4d8eed71dba56ed328c98cd4952d720
Normal-7611.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7611.png,Normal/images/Normal-7611.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7611.png,Normal/masks/Normal-7611.png,True,False,299,299,L,90.95,8a0af3ce6a188b4c266d01294d8b59a8
Normal-7612.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7612.png,Normal/images/Normal-7612.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7612.png,Normal/masks/Normal-7612.png,True,False,299,299,L,119.11,654fc642f1ce1456fbadee07c6e8b8f1
Normal-7613.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7613.png,Normal/images/Normal-7613.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7613.png,Normal/masks/Normal-7613.png,True,False,299,299,L,167.55,df4c7e752ded7056b5f457a361894d6c
Normal-7614.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7614.png,Normal/images/Normal-7614.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7614.png,Normal/masks/Normal-7614.png,True,False,299,299,L,129.25,879be35c380389852d370a306f30c98c
Normal-7615.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7615.png,Normal/images/Normal-7615.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7615.png,Normal/masks/Normal-7615.png,True,False,299,299,L,106.94,5d208e2bf4d3851834a03a18aa122f6e
Normal-7616.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7616.png,Normal/images/Normal-7616.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7616.png,Normal/masks/Normal-7616.png,True,False,299,299,L,123.83,f27e2d42dad0b23b82a35620f7bbf0c1
Normal-7617.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7617.png,Normal/images/Normal-7617.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7617.png,Normal/masks/Normal-7617.png,True,False,299,299,L,113.97,501cd53d0fc127e8d2def1f7d7fcae69
Normal-7618.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7618.png,Normal/images/Normal-7618.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7618.png,Normal/masks/Normal-7618.png,True,False,299,299,L,170.82,b05c9d6288b7a04b872733bde95fa6b2
Normal-7619.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7619.png,Normal/images/Normal-7619.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7619.png,Normal/masks/Normal-7619.png,True,False,299,299,L,97.1,ed86f4875de61cc7e3868cb868b65641
Normal-762.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-762.png,Normal/images/Normal-762.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-762.png,Normal/masks/Normal-762.png,True,False,299,299,L,124.92,7eeaf42b55dde7d8bac08e3f3ef9c9bc
Normal-7620.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7620.png,Normal/images/Normal-7620.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7620.png,Normal/masks/Normal-7620.png,True,False,299,299,L,138.31,2e87e84808c1ec38f4ad496f0c4171a5
Normal-7621.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7621.png,Normal/images/Normal-7621.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7621.png,Normal/masks/Normal-7621.png,True,False,299,299,L,189.59,33a20ec6aaeba111d1e3edc7ccf1e069
Normal-7622.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7622.png,Normal/images/Normal-7622.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7622.png,Normal/masks/Normal-7622.png,True,False,299,299,L,160.64,eec2f495d9a52e37a9203de7f600aef8
Normal-7623.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7623.png,Normal/images/Normal-7623.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7623.png,Normal/masks/Normal-7623.png,True,False,299,299,L,172.89,93d7c354e9fc084e1cbb5a03ef7abcd6
Normal-7624.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7624.png,Normal/images/Normal-7624.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7624.png,Normal/masks/Normal-7624.png,True,False,299,299,L,126.61,91191a03125c86cab13597208eb56a27
Normal-7625.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7625.png,Normal/images/Normal-7625.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7625.png,Normal/masks/Normal-7625.png,True,False,299,299,L,102.27,cfabd6c73e15200ab111c8be6d4c2180
Normal-7626.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7626.png,Normal/images/Normal-7626.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7626.png,Normal/masks/Normal-7626.png,True,False,299,299,L,108.34,ce71572d96b76a1438739f6763d7316c
Normal-7627.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7627.png,Normal/images/Normal-7627.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7627.png,Normal/masks/Normal-7627.png,True,False,299,299,L,135.23,6691ccd581d7ead8facc81c50c416846
Normal-7628.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7628.png,Normal/images/Normal-7628.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7628.png,Normal/masks/Normal-7628.png,True,False,299,299,L,132.12,75dfee9d2d705e47148fd75b939d3fda
Normal-7629.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7629.png,Normal/images/Normal-7629.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7629.png,Normal/masks/Normal-7629.png,True,False,299,299,L,127.74,0c88633b0b746e0ea38e88f030febca0
Normal-763.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-763.png,Normal/images/Normal-763.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-763.png,Normal/masks/Normal-763.png,True,False,299,299,L,112.35,e50836cc01470650216f5219535af62f
Normal-7630.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7630.png,Normal/images/Normal-7630.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7630.png,Normal/masks/Normal-7630.png,True,False,299,299,L,110.9,77e66ad3429cc706fe2f4ef4a2064eed
Normal-7631.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7631.png,Normal/images/Normal-7631.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7631.png,Normal/masks/Normal-7631.png,True,False,299,299,L,105.95,34615693b2c325a70b551d2cb885eefb
Normal-7632.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7632.png,Normal/images/Normal-7632.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7632.png,Normal/masks/Normal-7632.png,True,False,299,299,L,107.67,3e403d2115bde0edbd5d57bfbde9b843
Normal-7633.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7633.png,Normal/images/Normal-7633.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7633.png,Normal/masks/Normal-7633.png,True,False,299,299,L,104.14,75b7ea17d0ae1e3209b0c30012cdc89d
Normal-7634.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7634.png,Normal/images/Normal-7634.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7634.png,Normal/masks/Normal-7634.png,True,False,299,299,L,124.5,3583c5f8ce3ba08e1b22b3fe7be6cbd7
Normal-7635.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7635.png,Normal/images/Normal-7635.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7635.png,Normal/masks/Normal-7635.png,True,False,299,299,L,107.71,5afd6f24df840c229f1646557f2982dd
Normal-7636.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7636.png,Normal/images/Normal-7636.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7636.png,Normal/masks/Normal-7636.png,True,False,299,299,L,88.75,82bef94c6acb0ea7291f4848a8d956f3
Normal-7637.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7637.png,Normal/images/Normal-7637.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7637.png,Normal/masks/Normal-7637.png,True,False,299,299,L,93.73,8369d9e06a9b562db3af553d56b0c41c
Normal-7638.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7638.png,Normal/images/Normal-7638.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7638.png,Normal/masks/Normal-7638.png,True,False,299,299,L,173.54,d7c39160a325f116f14da65c5134180e
Normal-7639.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7639.png,Normal/images/Normal-7639.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7639.png,Normal/masks/Normal-7639.png,True,False,299,299,L,176.29,51f4ee15c1536f283cc9ef32f65d026b
Normal-764.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-764.png,Normal/images/Normal-764.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-764.png,Normal/masks/Normal-764.png,True,False,299,299,L,125.36,69dd3d12308863bb157c8777f478ddf7
Normal-7640.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7640.png,Normal/images/Normal-7640.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7640.png,Normal/masks/Normal-7640.png,True,False,299,299,L,115.28,5851f8633e6e3dfc3ccaffea99e6e70c
Normal-7641.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7641.png,Normal/images/Normal-7641.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7641.png,Normal/masks/Normal-7641.png,True,False,299,299,L,141.85,962846e9a84fa8493de9c13f130b7b69
Normal-7642.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7642.png,Normal/images/Normal-7642.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7642.png,Normal/masks/Normal-7642.png,True,False,299,299,L,141.81,e01e57a1b7ed2aa946653beeb9017156
Normal-7643.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7643.png,Normal/images/Normal-7643.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7643.png,Normal/masks/Normal-7643.png,True,False,299,299,L,142.76,57b5dfe3258772661afa6b6013a4c895
Normal-7644.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7644.png,Normal/images/Normal-7644.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7644.png,Normal/masks/Normal-7644.png,True,False,299,299,L,126.33,2de619f3b056b0503773f10b26cf067e
Normal-7645.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7645.png,Normal/images/Normal-7645.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7645.png,Normal/masks/Normal-7645.png,True,False,299,299,L,154.92,9c644fc476c31d5cb038e57aa69fc299
Normal-7646.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7646.png,Normal/images/Normal-7646.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7646.png,Normal/masks/Normal-7646.png,True,False,299,299,L,105.08,1277549bd72e6cb063fe6d902952c141
Normal-7647.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7647.png,Normal/images/Normal-7647.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7647.png,Normal/masks/Normal-7647.png,True,False,299,299,L,124.41,5cfd6f82cb05c274d28127f40f6c1212
Normal-7648.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7648.png,Normal/images/Normal-7648.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7648.png,Normal/masks/Normal-7648.png,True,False,299,299,L,123.61,0644e2ccdd9f1b2c58dd474fdd7beb45
Normal-7649.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7649.png,Normal/images/Normal-7649.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7649.png,Normal/masks/Normal-7649.png,True,False,299,299,L,123.7,74a45ee4698205442e6168aa5a4c161f
Normal-765.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-765.png,Normal/images/Normal-765.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-765.png,Normal/masks/Normal-765.png,True,False,299,299,L,120.39,6b6977e56ba420ceb1ebf8736456e2d9
Normal-7650.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7650.png,Normal/images/Normal-7650.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7650.png,Normal/masks/Normal-7650.png,True,False,299,299,L,160.0,03066e85844327a7efcd72cae3620d32
Normal-7651.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7651.png,Normal/images/Normal-7651.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7651.png,Normal/masks/Normal-7651.png,True,False,299,299,L,154.84,e12276c1341eaa3ae7dd05951aa9b7d7
Normal-7652.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7652.png,Normal/images/Normal-7652.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7652.png,Normal/masks/Normal-7652.png,True,False,299,299,L,117.89,6f2e993a56c43dc35f15cc1850d3fae8
Normal-7653.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7653.png,Normal/images/Normal-7653.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7653.png,Normal/masks/Normal-7653.png,True,False,299,299,L,108.94,c98351b921a794a1c62998e4948adf83
Normal-7654.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7654.png,Normal/images/Normal-7654.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7654.png,Normal/masks/Normal-7654.png,True,False,299,299,L,115.17,76bb302e3a34aba4dfca5c960ca0c5cd
Normal-7655.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7655.png,Normal/images/Normal-7655.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7655.png,Normal/masks/Normal-7655.png,True,False,299,299,L,119.06,bf85f77f4a00c3015c0053986420ac7b
Normal-7656.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7656.png,Normal/images/Normal-7656.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7656.png,Normal/masks/Normal-7656.png,True,False,299,299,L,131.49,c07cc96fedd5c00aa0c864010f473c86
Normal-7657.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7657.png,Normal/images/Normal-7657.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7657.png,Normal/masks/Normal-7657.png,True,False,299,299,L,141.68,b7d752072c5ccd0c51a0d524ecec3102
Normal-7658.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7658.png,Normal/images/Normal-7658.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7658.png,Normal/masks/Normal-7658.png,True,False,299,299,L,134.51,722f886f18206ac7c01e54069e752b5a
Normal-7659.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7659.png,Normal/images/Normal-7659.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7659.png,Normal/masks/Normal-7659.png,True,False,299,299,L,122.55,f159b5d19cfcbd5abee28cc11035da58
Normal-766.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-766.png,Normal/images/Normal-766.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-766.png,Normal/masks/Normal-766.png,True,False,299,299,L,110.21,79082358ed848b392a00786cc7403893
Normal-7660.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7660.png,Normal/images/Normal-7660.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7660.png,Normal/masks/Normal-7660.png,True,False,299,299,L,163.67,83b67bcb197e0b15fe4457cccea946df
Normal-7661.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7661.png,Normal/images/Normal-7661.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7661.png,Normal/masks/Normal-7661.png,True,False,299,299,L,150.02,43b858f84bfd96091ccaac88631a15c3
Normal-7662.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7662.png,Normal/images/Normal-7662.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7662.png,Normal/masks/Normal-7662.png,True,False,299,299,L,144.69,cd925b5ddb7fc2a60ea486bb7afb8cef
Normal-7663.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7663.png,Normal/images/Normal-7663.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7663.png,Normal/masks/Normal-7663.png,True,False,299,299,L,128.39,aa732ddb3bf13333af1988a650df2be5
Normal-7664.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7664.png,Normal/images/Normal-7664.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7664.png,Normal/masks/Normal-7664.png,True,False,299,299,L,109.01,e48c580ec93e413e3a25c36f9f15ea5f
Normal-7665.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7665.png,Normal/images/Normal-7665.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7665.png,Normal/masks/Normal-7665.png,True,False,299,299,L,120.1,9ea9e1be822ff43f289604e86b3c9a6c
Normal-7666.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7666.png,Normal/images/Normal-7666.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7666.png,Normal/masks/Normal-7666.png,True,False,299,299,L,170.36,5a4629dcee5b501032e00507a234cb35
Normal-7667.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7667.png,Normal/images/Normal-7667.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7667.png,Normal/masks/Normal-7667.png,True,False,299,299,L,117.62,19527aa377931c469c359fc10f56c762
Normal-7668.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7668.png,Normal/images/Normal-7668.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7668.png,Normal/masks/Normal-7668.png,True,False,299,299,L,117.26,15ff006dd2adbe8971bff012e4402bdb
Normal-7669.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7669.png,Normal/images/Normal-7669.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7669.png,Normal/masks/Normal-7669.png,True,False,299,299,L,106.78,7aea4f91301a68a7ddc2b6e98bb917bd
Normal-767.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-767.png,Normal/images/Normal-767.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-767.png,Normal/masks/Normal-767.png,True,False,299,299,L,116.55,1fb1031e66614be52d79c37f7e324e02
Normal-7670.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7670.png,Normal/images/Normal-7670.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7670.png,Normal/masks/Normal-7670.png,True,False,299,299,L,106.69,63b3e3c6090f93a2b702a79e5bb81bf7
Normal-7671.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7671.png,Normal/images/Normal-7671.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7671.png,Normal/masks/Normal-7671.png,True,False,299,299,L,117.22,836f06e7c103930ea77a6bb33cdb4709
Normal-7672.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7672.png,Normal/images/Normal-7672.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7672.png,Normal/masks/Normal-7672.png,True,False,299,299,L,180.56,cc42697ea73ac83f8e7b5e282764da58
Normal-7673.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7673.png,Normal/images/Normal-7673.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7673.png,Normal/masks/Normal-7673.png,True,False,299,299,L,131.83,c7208c008a05ab6236a008fa956af870
Normal-7674.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7674.png,Normal/images/Normal-7674.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7674.png,Normal/masks/Normal-7674.png,True,False,299,299,L,125.84,8f481678b114ef8b5285da43bbefa7cb
Normal-7675.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7675.png,Normal/images/Normal-7675.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7675.png,Normal/masks/Normal-7675.png,True,False,299,299,L,130.79,cbae63a9709ba7e8e7cf37d9a415c3d9
Normal-7676.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7676.png,Normal/images/Normal-7676.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7676.png,Normal/masks/Normal-7676.png,True,False,299,299,L,106.57,3f3f22db5878420a2e88237908a0f9bb
Normal-7677.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7677.png,Normal/images/Normal-7677.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7677.png,Normal/masks/Normal-7677.png,True,False,299,299,L,104.08,53bde7d2443e2a6b5b50fab1dee50117
Normal-7678.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7678.png,Normal/images/Normal-7678.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7678.png,Normal/masks/Normal-7678.png,True,False,299,299,L,121.4,9d14f83942d3faaa99dd7656872f8838
Normal-7679.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7679.png,Normal/images/Normal-7679.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7679.png,Normal/masks/Normal-7679.png,True,False,299,299,L,131.6,6f0eb7174e54bf87a9ff8a145734db70
Normal-768.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-768.png,Normal/images/Normal-768.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-768.png,Normal/masks/Normal-768.png,True,False,299,299,L,143.27,d626050ce44f93ea078ed2b4337c63e3
Normal-7680.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7680.png,Normal/images/Normal-7680.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7680.png,Normal/masks/Normal-7680.png,True,False,299,299,L,130.14,cd1e09cfb61347ba41105d1e7cc3472b
Normal-7681.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7681.png,Normal/images/Normal-7681.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7681.png,Normal/masks/Normal-7681.png,True,False,299,299,L,146.01,0bae90964a51f433aab3ff7c89d40d66
Normal-7682.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7682.png,Normal/images/Normal-7682.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7682.png,Normal/masks/Normal-7682.png,True,False,299,299,L,59.4,b22e72b65514619eca4e7e353ce5095a
Normal-7683.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7683.png,Normal/images/Normal-7683.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7683.png,Normal/masks/Normal-7683.png,True,False,299,299,L,160.44,6dcefe9cf3e55a834fbc01907969acec
Normal-7684.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7684.png,Normal/images/Normal-7684.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7684.png,Normal/masks/Normal-7684.png,True,False,299,299,L,129.74,4db226e01b2e0d0ef30a814f9ea11729
Normal-7685.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7685.png,Normal/images/Normal-7685.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7685.png,Normal/masks/Normal-7685.png,True,False,299,299,L,133.72,7db218e4803e96891c6ba9155d701123
Normal-7686.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7686.png,Normal/images/Normal-7686.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7686.png,Normal/masks/Normal-7686.png,True,False,299,299,L,160.88,e07076082314c998e166f9e060edff01
Normal-7687.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7687.png,Normal/images/Normal-7687.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7687.png,Normal/masks/Normal-7687.png,True,False,299,299,L,108.11,355449a14e5f47949dece5c8e98c0afd
Normal-7688.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7688.png,Normal/images/Normal-7688.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7688.png,Normal/masks/Normal-7688.png,True,False,299,299,L,130.94,1cfe3f58f829e446ae856afaf5a1a3fa
Normal-7689.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7689.png,Normal/images/Normal-7689.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7689.png,Normal/masks/Normal-7689.png,True,False,299,299,L,177.31,6671338a52e661328631c019a42b63cc
Normal-769.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-769.png,Normal/images/Normal-769.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-769.png,Normal/masks/Normal-769.png,True,False,299,299,L,143.88,dde753602953321805105c60de78c6a5
Normal-7690.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7690.png,Normal/images/Normal-7690.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7690.png,Normal/masks/Normal-7690.png,True,False,299,299,L,114.87,2044396f674dd1ff16ea1b91cbae7e78
Normal-7691.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7691.png,Normal/images/Normal-7691.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7691.png,Normal/masks/Normal-7691.png,True,False,299,299,L,105.86,6f6ae79252b5f0d64c64a1cab332fc9e
Normal-7692.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7692.png,Normal/images/Normal-7692.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7692.png,Normal/masks/Normal-7692.png,True,False,299,299,L,107.44,4c27fd2719106887fb5747dcc15d2a57
Normal-7693.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7693.png,Normal/images/Normal-7693.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7693.png,Normal/masks/Normal-7693.png,True,False,299,299,L,150.15,cdb2c1de3d75a25bcce77066a557d77e
Normal-7694.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7694.png,Normal/images/Normal-7694.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7694.png,Normal/masks/Normal-7694.png,True,False,299,299,L,129.35,137d7553fe8c89052f494e952a8cb7a3
Normal-7695.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7695.png,Normal/images/Normal-7695.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7695.png,Normal/masks/Normal-7695.png,True,False,299,299,L,111.93,99de6a15898649484904bc137626864f
Normal-7696.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7696.png,Normal/images/Normal-7696.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7696.png,Normal/masks/Normal-7696.png,True,False,299,299,L,103.76,8815f99ccce94cd4bc9f6e723f231997
Normal-7697.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7697.png,Normal/images/Normal-7697.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7697.png,Normal/masks/Normal-7697.png,True,False,299,299,L,108.4,bf865ddc03f1a9c32bab4a4d5cd8d6b3
Normal-7698.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7698.png,Normal/images/Normal-7698.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7698.png,Normal/masks/Normal-7698.png,True,False,299,299,L,99.07,4957bdab22b2d6026c7cab2a96e10dc1
Normal-7699.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7699.png,Normal/images/Normal-7699.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7699.png,Normal/masks/Normal-7699.png,True,False,299,299,L,126.59,6a8c000e41995a5932912b36e53be504
Normal-77.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-77.png,Normal/images/Normal-77.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-77.png,Normal/masks/Normal-77.png,True,False,299,299,L,126.43,279fc49bd178b4bc109273c1c8c8d5a8
Normal-770.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-770.png,Normal/images/Normal-770.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-770.png,Normal/masks/Normal-770.png,True,False,299,299,L,119.11,e12cb7f57acf1539ec20d656aad80597
Normal-7700.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7700.png,Normal/images/Normal-7700.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7700.png,Normal/masks/Normal-7700.png,True,False,299,299,L,126.2,113c70fbac08d151be50050b5607bd4c
Normal-7701.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7701.png,Normal/images/Normal-7701.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7701.png,Normal/masks/Normal-7701.png,True,False,299,299,L,163.23,e356a832bbee2868f05bd13eb46b1d2d
Normal-7702.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7702.png,Normal/images/Normal-7702.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7702.png,Normal/masks/Normal-7702.png,True,False,299,299,L,132.59,bdc270dd7318be8e0954549ba5748001
Normal-7703.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7703.png,Normal/images/Normal-7703.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7703.png,Normal/masks/Normal-7703.png,True,False,299,299,L,100.09,f94b9079e1a18022b64deb02e1a3e0b5
Normal-7704.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7704.png,Normal/images/Normal-7704.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7704.png,Normal/masks/Normal-7704.png,True,False,299,299,L,110.65,5993fa858fb9dcf605c7c07be95252e1
Normal-7705.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7705.png,Normal/images/Normal-7705.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7705.png,Normal/masks/Normal-7705.png,True,False,299,299,L,147.31,0b42aae28137bef190fafcc48d3b049e
Normal-7706.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7706.png,Normal/images/Normal-7706.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7706.png,Normal/masks/Normal-7706.png,True,False,299,299,L,165.74,bb1745a263602b2ef3f04b5c9fe99dcb
Normal-7707.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7707.png,Normal/images/Normal-7707.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7707.png,Normal/masks/Normal-7707.png,True,False,299,299,L,99.15,5574d6d92b2c62cbaf99e755d6b3c072
Normal-7708.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7708.png,Normal/images/Normal-7708.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7708.png,Normal/masks/Normal-7708.png,True,False,299,299,L,105.94,3cc7d07b971863ac089fa98b9fb74643
Normal-7709.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7709.png,Normal/images/Normal-7709.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7709.png,Normal/masks/Normal-7709.png,True,False,299,299,L,169.85,af00d38895f350442578ee4f3155c77d
Normal-771.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-771.png,Normal/images/Normal-771.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-771.png,Normal/masks/Normal-771.png,True,False,299,299,L,120.66,4897387d5c6f00109f386670a4dac8b0
Normal-7710.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7710.png,Normal/images/Normal-7710.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7710.png,Normal/masks/Normal-7710.png,True,False,299,299,L,134.82,4bcb9655a1967d5fcec05c52ad4eb646
Normal-7711.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7711.png,Normal/images/Normal-7711.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7711.png,Normal/masks/Normal-7711.png,True,False,299,299,L,109.99,4766b6387e34dfab0fd40547c6c1801a
Normal-7712.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7712.png,Normal/images/Normal-7712.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7712.png,Normal/masks/Normal-7712.png,True,False,299,299,L,164.43,45f674431ca70f24d039b909d7d5fdac
Normal-7713.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7713.png,Normal/images/Normal-7713.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7713.png,Normal/masks/Normal-7713.png,True,False,299,299,L,127.16,9f8bccd9098a759ed6f54b9683356cf7
Normal-7714.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7714.png,Normal/images/Normal-7714.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7714.png,Normal/masks/Normal-7714.png,True,False,299,299,L,117.41,5fd7b4b7d27571eeddb7aae838d5c79b
Normal-7715.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7715.png,Normal/images/Normal-7715.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7715.png,Normal/masks/Normal-7715.png,True,False,299,299,L,132.79,024a0631620c1fbc2bac80ba4a7e17d2
Normal-7716.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7716.png,Normal/images/Normal-7716.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7716.png,Normal/masks/Normal-7716.png,True,False,299,299,L,134.25,4a227f12801d20746bfd4f13edc67c1d
Normal-7717.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7717.png,Normal/images/Normal-7717.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7717.png,Normal/masks/Normal-7717.png,True,False,299,299,L,111.96,5d787d85f378c810d7b28980d5a876cd
Normal-7718.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7718.png,Normal/images/Normal-7718.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7718.png,Normal/masks/Normal-7718.png,True,False,299,299,L,161.92,f5dd9c78bfb5165ea29fd22e3529cb85
Normal-7719.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7719.png,Normal/images/Normal-7719.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7719.png,Normal/masks/Normal-7719.png,True,False,299,299,L,109.88,56282038c8cc4c769f8bcb120e4c2b62
Normal-772.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-772.png,Normal/images/Normal-772.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-772.png,Normal/masks/Normal-772.png,True,False,299,299,L,127.82,1fea3c8c6d5adfa885230f0a9f6eeb76
Normal-7720.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7720.png,Normal/images/Normal-7720.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7720.png,Normal/masks/Normal-7720.png,True,False,299,299,L,115.05,b42338bde18b076f71943ab2f26419be
Normal-7721.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7721.png,Normal/images/Normal-7721.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7721.png,Normal/masks/Normal-7721.png,True,False,299,299,L,99.02,3b32321b9288a28664666176fa6d8622
Normal-7722.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7722.png,Normal/images/Normal-7722.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7722.png,Normal/masks/Normal-7722.png,True,False,299,299,L,171.37,94e570e0366589fd1bcd02fbffff9732
Normal-7723.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7723.png,Normal/images/Normal-7723.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7723.png,Normal/masks/Normal-7723.png,True,False,299,299,L,106.79,0d9431ad9071d6f8e5b5e72ca6993252
Normal-7724.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7724.png,Normal/images/Normal-7724.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7724.png,Normal/masks/Normal-7724.png,True,False,299,299,L,105.98,6fa1db6b9ccd6d664a9ac64403692dfe
Normal-7725.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7725.png,Normal/images/Normal-7725.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7725.png,Normal/masks/Normal-7725.png,True,False,299,299,L,129.31,e79b3462c68111bc8561c21723c3842d
Normal-7726.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7726.png,Normal/images/Normal-7726.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7726.png,Normal/masks/Normal-7726.png,True,False,299,299,L,135.09,f912342e3658cb0dbd9b83b3c3a9b1a6
Normal-7727.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7727.png,Normal/images/Normal-7727.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7727.png,Normal/masks/Normal-7727.png,True,False,299,299,L,135.38,aadf9d20b6718c42ca5ca1af7b1e421c
Normal-7728.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7728.png,Normal/images/Normal-7728.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7728.png,Normal/masks/Normal-7728.png,True,False,299,299,L,149.15,a4733c72690a6144db291e1373334057
Normal-7729.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7729.png,Normal/images/Normal-7729.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7729.png,Normal/masks/Normal-7729.png,True,False,299,299,L,169.47,9acea44f81f373298d5020b52a741fc1
Normal-773.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-773.png,Normal/images/Normal-773.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-773.png,Normal/masks/Normal-773.png,True,False,299,299,L,129.74,cda788962ec46b63749918e2291bd4e2
Normal-7730.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7730.png,Normal/images/Normal-7730.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7730.png,Normal/masks/Normal-7730.png,True,False,299,299,L,153.19,4b673af6cf2037925c0715cfc14c0468
Normal-7731.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7731.png,Normal/images/Normal-7731.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7731.png,Normal/masks/Normal-7731.png,True,False,299,299,L,118.37,83e04170b9712c20d4256fe1f86ea54f
Normal-7732.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7732.png,Normal/images/Normal-7732.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7732.png,Normal/masks/Normal-7732.png,True,False,299,299,L,115.86,1394ab1248f1e767c95f1a170eaca1c5
Normal-7733.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7733.png,Normal/images/Normal-7733.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7733.png,Normal/masks/Normal-7733.png,True,False,299,299,L,100.1,b41e900c96179c71ea5b06ba2937ae9f
Normal-7734.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7734.png,Normal/images/Normal-7734.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7734.png,Normal/masks/Normal-7734.png,True,False,299,299,L,119.63,24850b596f88bc8a0265866b8e3c85d8
Normal-7735.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7735.png,Normal/images/Normal-7735.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7735.png,Normal/masks/Normal-7735.png,True,False,299,299,L,112.52,c6a5c6729bbb2540b7bc6635cc753d5c
Normal-7736.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7736.png,Normal/images/Normal-7736.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7736.png,Normal/masks/Normal-7736.png,True,False,299,299,L,135.18,8108a2095a35c9eaa04690857b3d671e
Normal-7737.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7737.png,Normal/images/Normal-7737.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7737.png,Normal/masks/Normal-7737.png,True,False,299,299,L,119.92,7750ee57a50b84c965587dc51ce26f82
Normal-7738.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7738.png,Normal/images/Normal-7738.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7738.png,Normal/masks/Normal-7738.png,True,False,299,299,L,95.44,6eabddee00ce7fef0293badbced2d901
Normal-7739.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7739.png,Normal/images/Normal-7739.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7739.png,Normal/masks/Normal-7739.png,True,False,299,299,L,123.32,2ed89d47aa68ba229adc2e7130b0f568
Normal-774.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-774.png,Normal/images/Normal-774.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-774.png,Normal/masks/Normal-774.png,True,False,299,299,L,96.2,bcdd60291cbdcc052c64db36c5da3d1b
Normal-7740.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7740.png,Normal/images/Normal-7740.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7740.png,Normal/masks/Normal-7740.png,True,False,299,299,L,100.78,0a38af92b6f6a9857bc1b4628c8a0709
Normal-7741.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7741.png,Normal/images/Normal-7741.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7741.png,Normal/masks/Normal-7741.png,True,False,299,299,L,104.28,80d074044147e2411ab00dfa5088ad15
Normal-7742.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7742.png,Normal/images/Normal-7742.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7742.png,Normal/masks/Normal-7742.png,True,False,299,299,L,121.85,a4dee905b98cc951d06ea5a6bf744286
Normal-7743.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7743.png,Normal/images/Normal-7743.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7743.png,Normal/masks/Normal-7743.png,True,False,299,299,L,108.53,1e86d74878f280219ed9c24d6716403a
Normal-7744.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7744.png,Normal/images/Normal-7744.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7744.png,Normal/masks/Normal-7744.png,True,False,299,299,L,134.04,3d524d97df6ef33a0d81c1f8c172de18
Normal-7745.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7745.png,Normal/images/Normal-7745.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7745.png,Normal/masks/Normal-7745.png,True,False,299,299,L,113.64,e1f988859c5c412b1652e39ed04d0ec8
Normal-7746.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7746.png,Normal/images/Normal-7746.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7746.png,Normal/masks/Normal-7746.png,True,False,299,299,L,125.08,d77adec7fe6f9609db83c839fe8393de
Normal-7747.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7747.png,Normal/images/Normal-7747.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7747.png,Normal/masks/Normal-7747.png,True,False,299,299,L,152.98,49cf94c977caf6d40e4899ee73d789d7
Normal-7748.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7748.png,Normal/images/Normal-7748.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7748.png,Normal/masks/Normal-7748.png,True,False,299,299,L,170.37,f62e32e4c0420715e3f985f5d00bae38
Normal-7749.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7749.png,Normal/images/Normal-7749.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7749.png,Normal/masks/Normal-7749.png,True,False,299,299,L,162.0,17565641ff3a38cd4112ba9500c35aca
Normal-775.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-775.png,Normal/images/Normal-775.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-775.png,Normal/masks/Normal-775.png,True,False,299,299,L,121.61,7d7c54f139c800eddc4b37a037317c46
Normal-7750.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7750.png,Normal/images/Normal-7750.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7750.png,Normal/masks/Normal-7750.png,True,False,299,299,L,137.69,9c13c4799d19ef6ff928e345f53dd0f1
Normal-7751.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7751.png,Normal/images/Normal-7751.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7751.png,Normal/masks/Normal-7751.png,True,False,299,299,L,168.17,7d9c308a85c3b4e1dba9ef59158e0e6c
Normal-7752.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7752.png,Normal/images/Normal-7752.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7752.png,Normal/masks/Normal-7752.png,True,False,299,299,L,134.9,637caaa15cca94a3ccf08e5773876a8f
Normal-7753.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7753.png,Normal/images/Normal-7753.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7753.png,Normal/masks/Normal-7753.png,True,False,299,299,L,109.84,f471aadd4999de0b90a90551ade1d0d5
Normal-7754.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7754.png,Normal/images/Normal-7754.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7754.png,Normal/masks/Normal-7754.png,True,False,299,299,L,166.74,f80ca116ad755719869875399e0a09ef
Normal-7755.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7755.png,Normal/images/Normal-7755.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7755.png,Normal/masks/Normal-7755.png,True,False,299,299,L,168.72,8450294caebb8f98952d4a604f3639ae
Normal-7756.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7756.png,Normal/images/Normal-7756.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7756.png,Normal/masks/Normal-7756.png,True,False,299,299,L,108.27,57f5a25204f823212895b7437c115d74
Normal-7757.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7757.png,Normal/images/Normal-7757.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7757.png,Normal/masks/Normal-7757.png,True,False,299,299,L,127.2,a958340539c6039187e2533909622859
Normal-7758.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7758.png,Normal/images/Normal-7758.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7758.png,Normal/masks/Normal-7758.png,True,False,299,299,L,114.09,ded7dba05c57814ff6b330e75add25c9
Normal-7759.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7759.png,Normal/images/Normal-7759.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7759.png,Normal/masks/Normal-7759.png,True,False,299,299,L,187.35,a8258a06de2193dd065ab3d674669319
Normal-776.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-776.png,Normal/images/Normal-776.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-776.png,Normal/masks/Normal-776.png,True,False,299,299,L,112.54,7198dec89a1b9ded6158d1e6463191b4
Normal-7760.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7760.png,Normal/images/Normal-7760.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7760.png,Normal/masks/Normal-7760.png,True,False,299,299,L,135.75,1d495b8cc7fe6927f3aae45768088765
Normal-7761.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7761.png,Normal/images/Normal-7761.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7761.png,Normal/masks/Normal-7761.png,True,False,299,299,L,117.9,32a1b4577743e1fbe931bcbeccf4ea0e
Normal-7762.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7762.png,Normal/images/Normal-7762.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7762.png,Normal/masks/Normal-7762.png,True,False,299,299,L,151.48,2095571860c522155204d6fbc2c0fd35
Normal-7763.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7763.png,Normal/images/Normal-7763.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7763.png,Normal/masks/Normal-7763.png,True,False,299,299,L,170.56,184a062c18b6b8fc2e95294872fb8519
Normal-7764.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7764.png,Normal/images/Normal-7764.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7764.png,Normal/masks/Normal-7764.png,True,False,299,299,L,134.72,1ec916bd8ac5289e77569c3225e12b8d
Normal-7765.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7765.png,Normal/images/Normal-7765.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7765.png,Normal/masks/Normal-7765.png,True,False,299,299,L,152.46,b9977d90714487ce487fc1eff7398be9
Normal-7766.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7766.png,Normal/images/Normal-7766.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7766.png,Normal/masks/Normal-7766.png,True,False,299,299,L,116.93,77ee781d526decdfc428cc1947249b6c
Normal-7767.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7767.png,Normal/images/Normal-7767.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7767.png,Normal/masks/Normal-7767.png,True,False,299,299,L,108.69,0b8e6dc1760e1302ee269e3bcda43618
Normal-7768.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7768.png,Normal/images/Normal-7768.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7768.png,Normal/masks/Normal-7768.png,True,False,299,299,L,106.89,be9f3c6be2dc3585b0011f199f36d4dd
Normal-7769.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7769.png,Normal/images/Normal-7769.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7769.png,Normal/masks/Normal-7769.png,True,False,299,299,L,107.96,38c3bf626724755b5351cb741a9640c6
Normal-777.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-777.png,Normal/images/Normal-777.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-777.png,Normal/masks/Normal-777.png,True,False,299,299,L,120.8,7cdeab141a9f395eb77695d7fec2bac5
Normal-7770.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7770.png,Normal/images/Normal-7770.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7770.png,Normal/masks/Normal-7770.png,True,False,299,299,L,171.62,5c9d2d3f2aa4034e8cd00ea26d23ad31
Normal-7771.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7771.png,Normal/images/Normal-7771.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7771.png,Normal/masks/Normal-7771.png,True,False,299,299,L,81.37,f4f3e5f39811d6aa8a9ca58f788065ef
Normal-7772.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7772.png,Normal/images/Normal-7772.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7772.png,Normal/masks/Normal-7772.png,True,False,299,299,L,147.25,f4f84d41b702f2d3abe39a1981227f26
Normal-7773.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7773.png,Normal/images/Normal-7773.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7773.png,Normal/masks/Normal-7773.png,True,False,299,299,L,133.55,c0bc23ca891d5a9c7a4c12ee0e65ae66
Normal-7774.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7774.png,Normal/images/Normal-7774.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7774.png,Normal/masks/Normal-7774.png,True,False,299,299,L,123.5,a340827b0fbe1cdfecb818bef3a35c13
Normal-7775.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7775.png,Normal/images/Normal-7775.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7775.png,Normal/masks/Normal-7775.png,True,False,299,299,L,162.66,7898983586588b0c7dbfa20d9ffc68de
Normal-7776.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7776.png,Normal/images/Normal-7776.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7776.png,Normal/masks/Normal-7776.png,True,False,299,299,L,141.03,b7caf87be1ac76241697ace2d9be9346
Normal-7777.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7777.png,Normal/images/Normal-7777.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7777.png,Normal/masks/Normal-7777.png,True,False,299,299,L,158.18,5f2cdeea35699eeba05c770a6503866b
Normal-7778.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7778.png,Normal/images/Normal-7778.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7778.png,Normal/masks/Normal-7778.png,True,False,299,299,L,110.4,0e1412f50d38c75dbdc002573d30ee50
Normal-7779.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7779.png,Normal/images/Normal-7779.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7779.png,Normal/masks/Normal-7779.png,True,False,299,299,L,166.26,e09a17631b148389857d240fd43cdd0e
Normal-778.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-778.png,Normal/images/Normal-778.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-778.png,Normal/masks/Normal-778.png,True,False,299,299,L,132.03,b7ac4c5717823b59201ed00bed01f6f5
Normal-7780.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7780.png,Normal/images/Normal-7780.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7780.png,Normal/masks/Normal-7780.png,True,False,299,299,L,101.67,7d9edd375f3d54a57de6349d65830cfc
Normal-7781.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7781.png,Normal/images/Normal-7781.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7781.png,Normal/masks/Normal-7781.png,True,False,299,299,L,167.13,afd0177427dbae67eb61ddf67bbe23d5
Normal-7782.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7782.png,Normal/images/Normal-7782.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7782.png,Normal/masks/Normal-7782.png,True,False,299,299,L,145.14,7be79dfd17bd2c853a164b38d27aef92
Normal-7783.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7783.png,Normal/images/Normal-7783.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7783.png,Normal/masks/Normal-7783.png,True,False,299,299,L,182.67,94aa9f1a279a47582eefee3ef10bd8fc
Normal-7784.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7784.png,Normal/images/Normal-7784.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7784.png,Normal/masks/Normal-7784.png,True,False,299,299,L,142.7,4fe29d3a563843c49c7f78b5e3618a81
Normal-7785.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7785.png,Normal/images/Normal-7785.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7785.png,Normal/masks/Normal-7785.png,True,False,299,299,L,161.49,9ef7503cea3bf621a2635f8df6e78a57
Normal-7786.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7786.png,Normal/images/Normal-7786.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7786.png,Normal/masks/Normal-7786.png,True,False,299,299,L,157.15,ee415d83628097ac8ffd96f3e0b627ce
Normal-7787.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7787.png,Normal/images/Normal-7787.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7787.png,Normal/masks/Normal-7787.png,True,False,299,299,L,168.28,07bb4f1d00374d9755af42b24d033a23
Normal-7788.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7788.png,Normal/images/Normal-7788.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7788.png,Normal/masks/Normal-7788.png,True,False,299,299,L,116.52,b563d83066342cac9d34e50b333b4e4a
Normal-7789.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7789.png,Normal/images/Normal-7789.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7789.png,Normal/masks/Normal-7789.png,True,False,299,299,L,104.33,3605db9fb872d0db8f796f7f6b694a5f
Normal-779.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-779.png,Normal/images/Normal-779.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-779.png,Normal/masks/Normal-779.png,True,False,299,299,L,115.32,0f207a5889467ac4f01db1866a769fd0
Normal-7790.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7790.png,Normal/images/Normal-7790.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7790.png,Normal/masks/Normal-7790.png,True,False,299,299,L,119.2,3314780a7786ba334c6f0a07093460a0
Normal-7791.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7791.png,Normal/images/Normal-7791.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7791.png,Normal/masks/Normal-7791.png,True,False,299,299,L,128.97,a9a9bbb69b3c3fd9a03b08ad7f936bd3
Normal-7792.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7792.png,Normal/images/Normal-7792.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7792.png,Normal/masks/Normal-7792.png,True,False,299,299,L,128.56,d59704f2e4a6e2b5114b0ba1876a1936
Normal-7793.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7793.png,Normal/images/Normal-7793.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7793.png,Normal/masks/Normal-7793.png,True,False,299,299,L,102.93,25b78d414ca66cafe10f5b03ce2de19c
Normal-7794.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7794.png,Normal/images/Normal-7794.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7794.png,Normal/masks/Normal-7794.png,True,False,299,299,L,120.28,9414d21cd93e6ceb58d98234e31228ff
Normal-7795.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7795.png,Normal/images/Normal-7795.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7795.png,Normal/masks/Normal-7795.png,True,False,299,299,L,114.07,56e1511d59f298b0bd952dd13f57a7cf
Normal-7796.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7796.png,Normal/images/Normal-7796.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7796.png,Normal/masks/Normal-7796.png,True,False,299,299,L,159.56,c70437c13949db611a64a97e281be61e
Normal-7797.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7797.png,Normal/images/Normal-7797.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7797.png,Normal/masks/Normal-7797.png,True,False,299,299,L,110.39,36d93fe1a3e8dbbeabbafda5beba0eaa
Normal-7798.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7798.png,Normal/images/Normal-7798.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7798.png,Normal/masks/Normal-7798.png,True,False,299,299,L,115.77,1161f2f0b027df71771c4823c7392ccb
Normal-7799.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7799.png,Normal/images/Normal-7799.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7799.png,Normal/masks/Normal-7799.png,True,False,299,299,L,117.72,d2c6134c056dd0dd07969197461198cf
Normal-78.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-78.png,Normal/images/Normal-78.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-78.png,Normal/masks/Normal-78.png,True,False,299,299,L,119.46,4b0280a0fd4b2154a497165b10e55011
Normal-780.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-780.png,Normal/images/Normal-780.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-780.png,Normal/masks/Normal-780.png,True,False,299,299,L,155.01,c9626ef42a1ff43c3a6333772174f039
Normal-7800.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7800.png,Normal/images/Normal-7800.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7800.png,Normal/masks/Normal-7800.png,True,False,299,299,L,118.13,e8274b1bd6525741a3a22fb85a070eb3
Normal-7801.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7801.png,Normal/images/Normal-7801.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7801.png,Normal/masks/Normal-7801.png,True,False,299,299,L,157.95,ed515762cd27fdc548169f96a111c0b0
Normal-7802.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7802.png,Normal/images/Normal-7802.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7802.png,Normal/masks/Normal-7802.png,True,False,299,299,L,115.82,c67dfaf250e9dcbc9fdb3322921df0eb
Normal-7803.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7803.png,Normal/images/Normal-7803.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7803.png,Normal/masks/Normal-7803.png,True,False,299,299,L,119.02,72ed1ea73c12e299e637d20548b0b8de
Normal-7804.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7804.png,Normal/images/Normal-7804.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7804.png,Normal/masks/Normal-7804.png,True,False,299,299,L,164.69,32eec868666f5a2818bba83ace7ce7d4
Normal-7805.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7805.png,Normal/images/Normal-7805.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7805.png,Normal/masks/Normal-7805.png,True,False,299,299,L,104.68,e100262c0190a5272c8a139b771c44c9
Normal-7806.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7806.png,Normal/images/Normal-7806.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7806.png,Normal/masks/Normal-7806.png,True,False,299,299,L,186.22,989d6f1ac7e3f57b3294528f3299c6a7
Normal-7807.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7807.png,Normal/images/Normal-7807.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7807.png,Normal/masks/Normal-7807.png,True,False,299,299,L,122.67,bad05ade99585e4dc5bd4f144dfa377a
Normal-7808.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7808.png,Normal/images/Normal-7808.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7808.png,Normal/masks/Normal-7808.png,True,False,299,299,L,166.61,02f0ad94b0b3a8b466b1c3792c3a33dd
Normal-7809.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7809.png,Normal/images/Normal-7809.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7809.png,Normal/masks/Normal-7809.png,True,False,299,299,L,171.27,75d057cb4dd27068edbe268c7be5c72c
Normal-781.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-781.png,Normal/images/Normal-781.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-781.png,Normal/masks/Normal-781.png,True,False,299,299,L,133.71,4ae8f02718b3025e31522d20df49f2ca
Normal-7810.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7810.png,Normal/images/Normal-7810.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7810.png,Normal/masks/Normal-7810.png,True,False,299,299,L,107.26,bf9ef9d19ca467365934ae0dfcd558b1
Normal-7811.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7811.png,Normal/images/Normal-7811.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7811.png,Normal/masks/Normal-7811.png,True,False,299,299,L,127.7,6b7a1d9de7d899d601ad7954fb321e2a
Normal-7812.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7812.png,Normal/images/Normal-7812.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7812.png,Normal/masks/Normal-7812.png,True,False,299,299,L,106.15,1c77a26bfffa108f27d8f7a277ac9730
Normal-7813.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7813.png,Normal/images/Normal-7813.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7813.png,Normal/masks/Normal-7813.png,True,False,299,299,L,134.41,35d9f89925f010939b5b20b7d7f3904f
Normal-7814.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7814.png,Normal/images/Normal-7814.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7814.png,Normal/masks/Normal-7814.png,True,False,299,299,L,176.5,2eefb4f2d25c7893babdcb49ad1ec25d
Normal-7815.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7815.png,Normal/images/Normal-7815.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7815.png,Normal/masks/Normal-7815.png,True,False,299,299,L,166.87,221df303920e8152fa4283cad81bffc5
Normal-7816.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7816.png,Normal/images/Normal-7816.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7816.png,Normal/masks/Normal-7816.png,True,False,299,299,L,165.05,4e085fb4508b41c543c2e55de70de4cc
Normal-7817.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7817.png,Normal/images/Normal-7817.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7817.png,Normal/masks/Normal-7817.png,True,False,299,299,L,128.6,116371260012eb50503d837ed3d39abb
Normal-7818.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7818.png,Normal/images/Normal-7818.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7818.png,Normal/masks/Normal-7818.png,True,False,299,299,L,169.46,bd741b2e5924b602f31894f08607db15
Normal-7819.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7819.png,Normal/images/Normal-7819.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7819.png,Normal/masks/Normal-7819.png,True,False,299,299,L,124.21,fb74ec41ceff57138b2dddc7ae09854e
Normal-782.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-782.png,Normal/images/Normal-782.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-782.png,Normal/masks/Normal-782.png,True,False,299,299,L,112.85,c8ddedd3f0ac1b493916dc8a6ea40a16
Normal-7820.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7820.png,Normal/images/Normal-7820.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7820.png,Normal/masks/Normal-7820.png,True,False,299,299,L,106.21,133df4c3cf36906b06e3f91007a51310
Normal-7821.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7821.png,Normal/images/Normal-7821.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7821.png,Normal/masks/Normal-7821.png,True,False,299,299,L,148.13,17d8f0643bd8e658e5d39a0b0af8f0dd
Normal-7822.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7822.png,Normal/images/Normal-7822.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7822.png,Normal/masks/Normal-7822.png,True,False,299,299,L,126.71,0328f6837f8a7373e51b43ad23e12763
Normal-7823.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7823.png,Normal/images/Normal-7823.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7823.png,Normal/masks/Normal-7823.png,True,False,299,299,L,125.88,c661acc399676c407b170e5a508bffb8
Normal-7824.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7824.png,Normal/images/Normal-7824.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7824.png,Normal/masks/Normal-7824.png,True,False,299,299,L,156.15,52ace48ff3969a495fde73e59464008e
Normal-7825.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7825.png,Normal/images/Normal-7825.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7825.png,Normal/masks/Normal-7825.png,True,False,299,299,L,100.0,cc0e8d7bb67535c2294dc0387f12c49d
Normal-7826.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7826.png,Normal/images/Normal-7826.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7826.png,Normal/masks/Normal-7826.png,True,False,299,299,L,137.41,86a611103f4121f89fe64696c336f674
Normal-7827.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7827.png,Normal/images/Normal-7827.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7827.png,Normal/masks/Normal-7827.png,True,False,299,299,L,118.32,aeead9ff1148e4d3c5df15858e763029
Normal-7828.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7828.png,Normal/images/Normal-7828.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7828.png,Normal/masks/Normal-7828.png,True,False,299,299,L,119.09,0737545afb1c55f6ba770c575c211faf
Normal-7829.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7829.png,Normal/images/Normal-7829.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7829.png,Normal/masks/Normal-7829.png,True,False,299,299,L,162.3,f853ce88505d7fedddd886886871d76f
Normal-783.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-783.png,Normal/images/Normal-783.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-783.png,Normal/masks/Normal-783.png,True,False,299,299,L,129.57,227ee464b28680d54edc98e3ecef934d
Normal-7830.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7830.png,Normal/images/Normal-7830.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7830.png,Normal/masks/Normal-7830.png,True,False,299,299,L,152.46,0330eeb16d69729deac55ef8959dae07
Normal-7831.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7831.png,Normal/images/Normal-7831.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7831.png,Normal/masks/Normal-7831.png,True,False,299,299,L,104.97,ecd4aab801b1b0491e56164507041534
Normal-7832.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7832.png,Normal/images/Normal-7832.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7832.png,Normal/masks/Normal-7832.png,True,False,299,299,L,129.03,802758066b6b97946d51ab38ca6d93c6
Normal-7833.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7833.png,Normal/images/Normal-7833.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7833.png,Normal/masks/Normal-7833.png,True,False,299,299,L,115.65,bb5d0ef8318217b7c69df6677a486c99
Normal-7834.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7834.png,Normal/images/Normal-7834.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7834.png,Normal/masks/Normal-7834.png,True,False,299,299,L,119.13,22d15ce1c60db029f65ef30a1fc19d17
Normal-7835.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7835.png,Normal/images/Normal-7835.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7835.png,Normal/masks/Normal-7835.png,True,False,299,299,L,111.23,92ac5db5245fab1d934e2faa6fb398a5
Normal-7836.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7836.png,Normal/images/Normal-7836.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7836.png,Normal/masks/Normal-7836.png,True,False,299,299,L,167.52,fc5b242e90280a51f15afa418ac161e2
Normal-7837.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7837.png,Normal/images/Normal-7837.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7837.png,Normal/masks/Normal-7837.png,True,False,299,299,L,146.53,aa504fd65cd39e3cb3033ae9c9399df4
Normal-7838.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7838.png,Normal/images/Normal-7838.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7838.png,Normal/masks/Normal-7838.png,True,False,299,299,L,110.62,e2c341ba45df5c997c62c81665c50c7e
Normal-7839.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7839.png,Normal/images/Normal-7839.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7839.png,Normal/masks/Normal-7839.png,True,False,299,299,L,131.79,c5b6c9fd32b2342566fa6e0463ca9b30
Normal-784.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-784.png,Normal/images/Normal-784.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-784.png,Normal/masks/Normal-784.png,True,False,299,299,L,130.04,fd3d6546d43bbc53756d7cdff78bb719
Normal-7840.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7840.png,Normal/images/Normal-7840.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7840.png,Normal/masks/Normal-7840.png,True,False,299,299,L,114.05,05854edfc438daed64cab72406fd6878
Normal-7841.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7841.png,Normal/images/Normal-7841.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7841.png,Normal/masks/Normal-7841.png,True,False,299,299,L,118.47,e871697ed4706630dcaf3b6228679f04
Normal-7842.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7842.png,Normal/images/Normal-7842.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7842.png,Normal/masks/Normal-7842.png,True,False,299,299,L,130.21,027ad595a55d855b06def0f8b9bfad12
Normal-7843.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7843.png,Normal/images/Normal-7843.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7843.png,Normal/masks/Normal-7843.png,True,False,299,299,L,118.0,93b92e7b7322ade0262181a06dc1e630
Normal-7844.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7844.png,Normal/images/Normal-7844.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7844.png,Normal/masks/Normal-7844.png,True,False,299,299,L,120.94,13211a8d67925d722b9642ba85c21b11
Normal-7845.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7845.png,Normal/images/Normal-7845.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7845.png,Normal/masks/Normal-7845.png,True,False,299,299,L,179.39,68333e295e2ce2174c4d37ed0137a92f
Normal-7846.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7846.png,Normal/images/Normal-7846.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7846.png,Normal/masks/Normal-7846.png,True,False,299,299,L,115.06,accb5f502fa3b58af924e467232f536a
Normal-7847.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7847.png,Normal/images/Normal-7847.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7847.png,Normal/masks/Normal-7847.png,True,False,299,299,L,128.21,03ad33f0f724da025966cc9cf0262aff
Normal-7848.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7848.png,Normal/images/Normal-7848.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7848.png,Normal/masks/Normal-7848.png,True,False,299,299,L,101.08,97544ad6db727eec792a0ca74307fd06
Normal-7849.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7849.png,Normal/images/Normal-7849.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7849.png,Normal/masks/Normal-7849.png,True,False,299,299,L,133.2,0b82118b096ec17aeb42eafa0a2fb501
Normal-785.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-785.png,Normal/images/Normal-785.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-785.png,Normal/masks/Normal-785.png,True,False,299,299,L,116.37,aa2daa4924ae873e1a58564d935ffe62
Normal-7850.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7850.png,Normal/images/Normal-7850.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7850.png,Normal/masks/Normal-7850.png,True,False,299,299,L,116.6,42bf9a6d82015d3fa115821bc4a132f6
Normal-7851.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7851.png,Normal/images/Normal-7851.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7851.png,Normal/masks/Normal-7851.png,True,False,299,299,L,150.86,f8beebb10033ae0e136aae87ff4427b5
Normal-7852.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7852.png,Normal/images/Normal-7852.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7852.png,Normal/masks/Normal-7852.png,True,False,299,299,L,172.95,d307b565deeffb24c3b10bf0cd53bfd0
Normal-7853.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7853.png,Normal/images/Normal-7853.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7853.png,Normal/masks/Normal-7853.png,True,False,299,299,L,130.34,95b6ece1e8a8527a51ed7f97c0dd22b7
Normal-7854.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7854.png,Normal/images/Normal-7854.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7854.png,Normal/masks/Normal-7854.png,True,False,299,299,L,133.82,bfe823dc2bab3f93c864d1e9c428758e
Normal-7855.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7855.png,Normal/images/Normal-7855.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7855.png,Normal/masks/Normal-7855.png,True,False,299,299,L,92.08,0fb89638d58ec10f68d652ee3cd200e2
Normal-7856.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7856.png,Normal/images/Normal-7856.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7856.png,Normal/masks/Normal-7856.png,True,False,299,299,L,116.04,052fa9ad989b54a5c1eab82e2d528dad
Normal-7857.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7857.png,Normal/images/Normal-7857.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7857.png,Normal/masks/Normal-7857.png,True,False,299,299,L,109.16,4aefbcff65c15858546683396a774cb4
Normal-7858.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7858.png,Normal/images/Normal-7858.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7858.png,Normal/masks/Normal-7858.png,True,False,299,299,L,116.26,18082b2bc896fa196a5dde806632c75d
Normal-7859.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7859.png,Normal/images/Normal-7859.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7859.png,Normal/masks/Normal-7859.png,True,False,299,299,L,143.08,d1bda6a3d5bf8ca75ef044cb2e7231e7
Normal-786.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-786.png,Normal/images/Normal-786.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-786.png,Normal/masks/Normal-786.png,True,False,299,299,L,119.47,e6832fb786717e275863e5615b76257a
Normal-7860.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7860.png,Normal/images/Normal-7860.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7860.png,Normal/masks/Normal-7860.png,True,False,299,299,L,132.44,6610b9a1df3cd6b3054e1cfe055a41fc
Normal-7861.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7861.png,Normal/images/Normal-7861.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7861.png,Normal/masks/Normal-7861.png,True,False,299,299,L,92.76,6b46b9e66562bbe3588fcfb835387c11
Normal-7862.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7862.png,Normal/images/Normal-7862.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7862.png,Normal/masks/Normal-7862.png,True,False,299,299,L,104.82,44498c296bf2a94756f86329a9c5251a
Normal-7863.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7863.png,Normal/images/Normal-7863.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7863.png,Normal/masks/Normal-7863.png,True,False,299,299,L,121.74,87fefcab2d8401edd65164a7b4b43bd9
Normal-7864.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7864.png,Normal/images/Normal-7864.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7864.png,Normal/masks/Normal-7864.png,True,False,299,299,L,168.07,855c5ea84f6f6c3db3d32a68caa69778
Normal-7865.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7865.png,Normal/images/Normal-7865.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7865.png,Normal/masks/Normal-7865.png,True,False,299,299,L,117.67,3d1b2df9c9b37b47220610f31b4a8801
Normal-7866.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7866.png,Normal/images/Normal-7866.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7866.png,Normal/masks/Normal-7866.png,True,False,299,299,L,133.32,8dcdcda288306cc51f69819c641f5873
Normal-7867.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7867.png,Normal/images/Normal-7867.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7867.png,Normal/masks/Normal-7867.png,True,False,299,299,L,181.6,a970bae4a2be7e7395d0b1c9b5aac792
Normal-7868.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7868.png,Normal/images/Normal-7868.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7868.png,Normal/masks/Normal-7868.png,True,False,299,299,L,97.7,d08c6281b3ac075d60c98a0e8bba496d
Normal-7869.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7869.png,Normal/images/Normal-7869.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7869.png,Normal/masks/Normal-7869.png,True,False,299,299,L,144.23,19260f163c3c6a47da1f7a641a951271
Normal-787.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-787.png,Normal/images/Normal-787.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-787.png,Normal/masks/Normal-787.png,True,False,299,299,L,132.81,7f5bc055ce89a0e1960d1f3acdbd2269
Normal-7870.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7870.png,Normal/images/Normal-7870.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7870.png,Normal/masks/Normal-7870.png,True,False,299,299,L,145.72,ad7631426ad0676d18f27afeca37d950
Normal-7871.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7871.png,Normal/images/Normal-7871.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7871.png,Normal/masks/Normal-7871.png,True,False,299,299,L,127.68,98033734debe96ea281a671d3ebce169
Normal-7872.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7872.png,Normal/images/Normal-7872.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7872.png,Normal/masks/Normal-7872.png,True,False,299,299,L,119.36,024ca1398dfc513b04cd3f4fb6ded3b0
Normal-7873.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7873.png,Normal/images/Normal-7873.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7873.png,Normal/masks/Normal-7873.png,True,False,299,299,L,109.66,6f248371f23cea9524be6678ee0c24c5
Normal-7874.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7874.png,Normal/images/Normal-7874.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7874.png,Normal/masks/Normal-7874.png,True,False,299,299,L,114.34,b1c32c9edc9a36f339b17531a4a42bf1
Normal-7875.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7875.png,Normal/images/Normal-7875.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7875.png,Normal/masks/Normal-7875.png,True,False,299,299,L,136.77,6de3bc2777160f0b918e3b6132a5c255
Normal-7876.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7876.png,Normal/images/Normal-7876.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7876.png,Normal/masks/Normal-7876.png,True,False,299,299,L,163.02,6d2f7eeb163cc308bc8b38fe146512db
Normal-7877.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7877.png,Normal/images/Normal-7877.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7877.png,Normal/masks/Normal-7877.png,True,False,299,299,L,113.45,e749ce8119da6784ea8001970ce5616d
Normal-7878.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7878.png,Normal/images/Normal-7878.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7878.png,Normal/masks/Normal-7878.png,True,False,299,299,L,131.7,ab2a1d64bea49fbb78a34e3b075abaa2
Normal-7879.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7879.png,Normal/images/Normal-7879.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7879.png,Normal/masks/Normal-7879.png,True,False,299,299,L,145.76,753b16e618f8bbedffef6a959056576f
Normal-788.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-788.png,Normal/images/Normal-788.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-788.png,Normal/masks/Normal-788.png,True,False,299,299,L,128.73,5e7b767948980456f93bf29361ebbeb0
Normal-7880.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7880.png,Normal/images/Normal-7880.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7880.png,Normal/masks/Normal-7880.png,True,False,299,299,L,99.84,11639b40100a4057e4db34b19e78012c
Normal-7881.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7881.png,Normal/images/Normal-7881.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7881.png,Normal/masks/Normal-7881.png,True,False,299,299,L,96.39,1eebf47b7233493ba5f3df1e3a042fb9
Normal-7882.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7882.png,Normal/images/Normal-7882.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7882.png,Normal/masks/Normal-7882.png,True,False,299,299,L,116.64,24ee4b126ffe2b50dafcdd18814d0100
Normal-7883.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7883.png,Normal/images/Normal-7883.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7883.png,Normal/masks/Normal-7883.png,True,False,299,299,L,174.93,5a3e99f25e0134139dcc99747ed95eb9
Normal-7884.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7884.png,Normal/images/Normal-7884.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7884.png,Normal/masks/Normal-7884.png,True,False,299,299,L,128.29,760f9f351742f21b1244c63d2b48ab21
Normal-7885.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7885.png,Normal/images/Normal-7885.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7885.png,Normal/masks/Normal-7885.png,True,False,299,299,L,119.99,b6e2a5327251b69d277f8d9396fc0981
Normal-7886.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7886.png,Normal/images/Normal-7886.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7886.png,Normal/masks/Normal-7886.png,True,False,299,299,L,112.55,2782a1dac58b8fa159acd4ef36b1835a
Normal-7887.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7887.png,Normal/images/Normal-7887.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7887.png,Normal/masks/Normal-7887.png,True,False,299,299,L,149.95,b0075c0f7cd47ef7ff19093f33bcbce9
Normal-7888.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7888.png,Normal/images/Normal-7888.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7888.png,Normal/masks/Normal-7888.png,True,False,299,299,L,112.39,654c49cb78c386928000e944552b1c1a
Normal-7889.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7889.png,Normal/images/Normal-7889.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7889.png,Normal/masks/Normal-7889.png,True,False,299,299,L,107.43,5e4d02650f1e250b12e0235608a7b958
Normal-789.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-789.png,Normal/images/Normal-789.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-789.png,Normal/masks/Normal-789.png,True,False,299,299,L,118.26,babe58ccbcb4ef974eb97125c90e439e
Normal-7890.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7890.png,Normal/images/Normal-7890.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7890.png,Normal/masks/Normal-7890.png,True,False,299,299,L,122.43,88f2b1f35a39ea2f3216a35b59947e74
Normal-7891.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7891.png,Normal/images/Normal-7891.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7891.png,Normal/masks/Normal-7891.png,True,False,299,299,L,94.79,5389e12063132913fda7c5100964c1bc
Normal-7892.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7892.png,Normal/images/Normal-7892.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7892.png,Normal/masks/Normal-7892.png,True,False,299,299,L,109.99,c72a822e497d090b58381e33d420e453
Normal-7893.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7893.png,Normal/images/Normal-7893.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7893.png,Normal/masks/Normal-7893.png,True,False,299,299,L,137.7,cc49b433ca9424669dfd916bfaa42d31
Normal-7894.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7894.png,Normal/images/Normal-7894.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7894.png,Normal/masks/Normal-7894.png,True,False,299,299,L,163.26,c19e9c65979af1aa6e9756bfbca26e13
Normal-7895.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7895.png,Normal/images/Normal-7895.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7895.png,Normal/masks/Normal-7895.png,True,False,299,299,L,101.28,bdb7cb8edb541526bc7f5c7ade239e86
Normal-7896.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7896.png,Normal/images/Normal-7896.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7896.png,Normal/masks/Normal-7896.png,True,False,299,299,L,124.87,5ae8f049e32d530b4ba6ee23d9b37eee
Normal-7897.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7897.png,Normal/images/Normal-7897.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7897.png,Normal/masks/Normal-7897.png,True,False,299,299,L,162.42,e8df393107e2529813e9f5b3debec0f3
Normal-7898.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7898.png,Normal/images/Normal-7898.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7898.png,Normal/masks/Normal-7898.png,True,False,299,299,L,152.2,61e3e616c34b4404d9d779488b40bce9
Normal-7899.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7899.png,Normal/images/Normal-7899.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7899.png,Normal/masks/Normal-7899.png,True,False,299,299,L,118.08,ff428bf47aff8941a0093945b1d1d088
Normal-79.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-79.png,Normal/images/Normal-79.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-79.png,Normal/masks/Normal-79.png,True,False,299,299,L,117.04,e2bb61fd8ee6138b2a496698fa217e25
Normal-790.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-790.png,Normal/images/Normal-790.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-790.png,Normal/masks/Normal-790.png,True,False,299,299,L,132.3,e8a8ea7bea2d4057dca822309c27dcff
Normal-7900.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7900.png,Normal/images/Normal-7900.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7900.png,Normal/masks/Normal-7900.png,True,False,299,299,L,87.79,1518cb52e0db97916a79d30a01072714
Normal-7901.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7901.png,Normal/images/Normal-7901.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7901.png,Normal/masks/Normal-7901.png,True,False,299,299,L,161.68,048e2830c898d6114c9edb7c3d1a8c07
Normal-7902.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7902.png,Normal/images/Normal-7902.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7902.png,Normal/masks/Normal-7902.png,True,False,299,299,L,153.64,ae8f8f95e7649caded7ca19d9d9920e5
Normal-7903.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7903.png,Normal/images/Normal-7903.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7903.png,Normal/masks/Normal-7903.png,True,False,299,299,L,126.53,610ecb2b052ee4dcf7e380424e0dc997
Normal-7904.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7904.png,Normal/images/Normal-7904.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7904.png,Normal/masks/Normal-7904.png,True,False,299,299,L,141.65,e15ba3a551baa1ec1779f01f8fd00dce
Normal-7905.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7905.png,Normal/images/Normal-7905.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7905.png,Normal/masks/Normal-7905.png,True,False,299,299,L,160.58,1cb374058eaf98290a2a543f2206ac1c
Normal-7906.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7906.png,Normal/images/Normal-7906.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7906.png,Normal/masks/Normal-7906.png,True,False,299,299,L,186.12,579373c05b8b1cd13780f3e2132dbf5f
Normal-7907.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7907.png,Normal/images/Normal-7907.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7907.png,Normal/masks/Normal-7907.png,True,False,299,299,L,119.51,2f60e51ffd798bc54d9af5ecbd39a06b
Normal-7908.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7908.png,Normal/images/Normal-7908.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7908.png,Normal/masks/Normal-7908.png,True,False,299,299,L,127.43,c1970337ab5bde8b332e3dc5e71249a9
Normal-7909.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7909.png,Normal/images/Normal-7909.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7909.png,Normal/masks/Normal-7909.png,True,False,299,299,L,92.46,e1ef0253de8649085bfc95935c85a093
Normal-791.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-791.png,Normal/images/Normal-791.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-791.png,Normal/masks/Normal-791.png,True,False,299,299,L,140.1,6d11b0bcf6396c2b71eba04b75478c7c
Normal-7910.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7910.png,Normal/images/Normal-7910.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7910.png,Normal/masks/Normal-7910.png,True,False,299,299,L,170.11,1eaf0d321a1b92567060a49726b7cbdd
Normal-7911.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7911.png,Normal/images/Normal-7911.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7911.png,Normal/masks/Normal-7911.png,True,False,299,299,L,116.59,d3870ab2091048ae845a2c5d2c3bf003
Normal-7912.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7912.png,Normal/images/Normal-7912.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7912.png,Normal/masks/Normal-7912.png,True,False,299,299,L,138.61,26ac7a396e9da6af747ad901d84a6b32
Normal-7913.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7913.png,Normal/images/Normal-7913.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7913.png,Normal/masks/Normal-7913.png,True,False,299,299,L,149.71,610c06ec03e84badb62b948d2ccf8903
Normal-7914.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7914.png,Normal/images/Normal-7914.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7914.png,Normal/masks/Normal-7914.png,True,False,299,299,L,108.24,8fccd41d802f69e606b746bfdc70c78a
Normal-7915.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7915.png,Normal/images/Normal-7915.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7915.png,Normal/masks/Normal-7915.png,True,False,299,299,L,110.65,ee06ee4302fa56a289aa08f8d9f72302
Normal-7916.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7916.png,Normal/images/Normal-7916.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7916.png,Normal/masks/Normal-7916.png,True,False,299,299,L,163.59,64407f3f1777627806f1dbaebaf083ea
Normal-7917.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7917.png,Normal/images/Normal-7917.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7917.png,Normal/masks/Normal-7917.png,True,False,299,299,L,104.56,a3bcf5ee843c6bada5f177dc6793a82c
Normal-7918.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7918.png,Normal/images/Normal-7918.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7918.png,Normal/masks/Normal-7918.png,True,False,299,299,L,106.0,7115751c4d2a98ca2a931949819bd9a6
Normal-7919.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7919.png,Normal/images/Normal-7919.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7919.png,Normal/masks/Normal-7919.png,True,False,299,299,L,134.25,e5a0e64143392ef350cd706197327e41
Normal-792.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-792.png,Normal/images/Normal-792.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-792.png,Normal/masks/Normal-792.png,True,False,299,299,L,115.97,cebb7fc552569e920bad33c234de0cd6
Normal-7920.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7920.png,Normal/images/Normal-7920.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7920.png,Normal/masks/Normal-7920.png,True,False,299,299,L,127.51,83703f932cfaef63c4dc82121cddfe22
Normal-7921.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7921.png,Normal/images/Normal-7921.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7921.png,Normal/masks/Normal-7921.png,True,False,299,299,L,148.53,4ef0486b81081bf9557bcc9a753de133
Normal-7922.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7922.png,Normal/images/Normal-7922.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7922.png,Normal/masks/Normal-7922.png,True,False,299,299,L,157.4,25056c3a2ad2136cfd696fbe15a195f9
Normal-7923.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7923.png,Normal/images/Normal-7923.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7923.png,Normal/masks/Normal-7923.png,True,False,299,299,L,104.61,7ff5eb724c327fefec34251531d07faa
Normal-7924.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7924.png,Normal/images/Normal-7924.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7924.png,Normal/masks/Normal-7924.png,True,False,299,299,L,107.82,c913c500d8fbbc47a90a9981df3d6364
Normal-7925.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7925.png,Normal/images/Normal-7925.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7925.png,Normal/masks/Normal-7925.png,True,False,299,299,L,131.77,05c962ace3432613773f31d4bc5a7d06
Normal-7926.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7926.png,Normal/images/Normal-7926.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7926.png,Normal/masks/Normal-7926.png,True,False,299,299,L,114.65,2aeba1f0233d8f03318af123cd43278b
Normal-7927.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7927.png,Normal/images/Normal-7927.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7927.png,Normal/masks/Normal-7927.png,True,False,299,299,L,105.57,26c43f4aaf1c26fcc962fb6de8804d92
Normal-7928.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7928.png,Normal/images/Normal-7928.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7928.png,Normal/masks/Normal-7928.png,True,False,299,299,L,136.29,21144f3af508bf46a4b313e951c2982b
Normal-7929.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7929.png,Normal/images/Normal-7929.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7929.png,Normal/masks/Normal-7929.png,True,False,299,299,L,109.18,8b3af4f241ddb52ebe5f94002f35ffe8
Normal-793.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-793.png,Normal/images/Normal-793.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-793.png,Normal/masks/Normal-793.png,True,False,299,299,L,110.51,ea9c75f2f76214a9c2f17f23c560474d
Normal-7930.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7930.png,Normal/images/Normal-7930.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7930.png,Normal/masks/Normal-7930.png,True,False,299,299,L,145.99,fe8a0e1daf97fa6025041aedc3a0331a
Normal-7931.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7931.png,Normal/images/Normal-7931.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7931.png,Normal/masks/Normal-7931.png,True,False,299,299,L,132.95,1435413ec2821899430fae05c386a6f6
Normal-7932.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7932.png,Normal/images/Normal-7932.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7932.png,Normal/masks/Normal-7932.png,True,False,299,299,L,123.31,552499e0a3c5cbee6385622a39a48693
Normal-7933.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7933.png,Normal/images/Normal-7933.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7933.png,Normal/masks/Normal-7933.png,True,False,299,299,L,113.93,5a6b1eaaabeed0bbf4f92b00d3b6f07a
Normal-7934.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7934.png,Normal/images/Normal-7934.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7934.png,Normal/masks/Normal-7934.png,True,False,299,299,L,106.7,2cae3639eb0693b90ecc1af3f5db7916
Normal-7935.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7935.png,Normal/images/Normal-7935.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7935.png,Normal/masks/Normal-7935.png,True,False,299,299,L,164.45,b283b783292479d92c0c67df445ab38e
Normal-7936.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7936.png,Normal/images/Normal-7936.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7936.png,Normal/masks/Normal-7936.png,True,False,299,299,L,155.36,8cfd291cd1ff8dcb0c458ed8afc52351
Normal-7937.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7937.png,Normal/images/Normal-7937.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7937.png,Normal/masks/Normal-7937.png,True,False,299,299,L,148.16,f9d4351a9e8de2723caf543e552e48ad
Normal-7938.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7938.png,Normal/images/Normal-7938.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7938.png,Normal/masks/Normal-7938.png,True,False,299,299,L,131.32,fa01563578aea0dbaf736e5e953486c6
Normal-7939.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7939.png,Normal/images/Normal-7939.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7939.png,Normal/masks/Normal-7939.png,True,False,299,299,L,111.36,d88a17a0c5e031459c31a251be2a961c
Normal-794.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-794.png,Normal/images/Normal-794.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-794.png,Normal/masks/Normal-794.png,True,False,299,299,L,122.52,ddec6131eb1fbd9f4467ab14a578f7e5
Normal-7940.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7940.png,Normal/images/Normal-7940.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7940.png,Normal/masks/Normal-7940.png,True,False,299,299,L,102.46,1a9b137247b795b21830ff8391690c14
Normal-7941.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7941.png,Normal/images/Normal-7941.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7941.png,Normal/masks/Normal-7941.png,True,False,299,299,L,147.9,92d90b3b73f4a61c9f50827b07388243
Normal-7942.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7942.png,Normal/images/Normal-7942.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7942.png,Normal/masks/Normal-7942.png,True,False,299,299,L,140.0,e3974ff4f0d97ce2588bacfa380b44d3
Normal-7943.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7943.png,Normal/images/Normal-7943.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7943.png,Normal/masks/Normal-7943.png,True,False,299,299,L,97.78,b4e39400445c5ffa84bb2bf7404497b4
Normal-7944.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7944.png,Normal/images/Normal-7944.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7944.png,Normal/masks/Normal-7944.png,True,False,299,299,L,107.22,aaee12d84c2be7de5e8e363dd10102c9
Normal-7945.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7945.png,Normal/images/Normal-7945.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7945.png,Normal/masks/Normal-7945.png,True,False,299,299,L,110.93,d6d711c1e2ccce89e6c5de63b5f33edd
Normal-7946.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7946.png,Normal/images/Normal-7946.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7946.png,Normal/masks/Normal-7946.png,True,False,299,299,L,171.39,1410def72b290725cca5c597d5560a72
Normal-7947.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7947.png,Normal/images/Normal-7947.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7947.png,Normal/masks/Normal-7947.png,True,False,299,299,L,135.68,f1420c4c0705bdd7ed235562a0544324
Normal-7948.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7948.png,Normal/images/Normal-7948.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7948.png,Normal/masks/Normal-7948.png,True,False,299,299,L,98.33,96b948043fcf9f063a304c712f4b2bbd
Normal-7949.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7949.png,Normal/images/Normal-7949.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7949.png,Normal/masks/Normal-7949.png,True,False,299,299,L,122.97,a3253ac6b8e11bbc48e2f2db5ede41ce
Normal-795.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-795.png,Normal/images/Normal-795.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-795.png,Normal/masks/Normal-795.png,True,False,299,299,L,122.64,a7cd7026ab836357752ce36bd7b42510
Normal-7950.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7950.png,Normal/images/Normal-7950.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7950.png,Normal/masks/Normal-7950.png,True,False,299,299,L,142.12,c8b2cf18ad332c7c00f3b14c52991e8b
Normal-7951.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7951.png,Normal/images/Normal-7951.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7951.png,Normal/masks/Normal-7951.png,True,False,299,299,L,131.56,a7b53f6d6d41147e51be813b311eb39e
Normal-7952.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7952.png,Normal/images/Normal-7952.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7952.png,Normal/masks/Normal-7952.png,True,False,299,299,L,112.98,8ff7e6e60c43aa06f5604148dd75a80f
Normal-7953.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7953.png,Normal/images/Normal-7953.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7953.png,Normal/masks/Normal-7953.png,True,False,299,299,L,112.05,3ac772a6248c1eb0413d8e9ec8bfc53d
Normal-7954.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7954.png,Normal/images/Normal-7954.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7954.png,Normal/masks/Normal-7954.png,True,False,299,299,L,105.53,64e18af2c3bdcb36bd56782c1692fda2
Normal-7955.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7955.png,Normal/images/Normal-7955.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7955.png,Normal/masks/Normal-7955.png,True,False,299,299,L,133.1,485b5115c660443b3856a10f115372d2
Normal-7956.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7956.png,Normal/images/Normal-7956.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7956.png,Normal/masks/Normal-7956.png,True,False,299,299,L,140.66,36687e401e5837f936a8d091ccfe41e3
Normal-7957.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7957.png,Normal/images/Normal-7957.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7957.png,Normal/masks/Normal-7957.png,True,False,299,299,L,144.16,ef7af814f806f99722fb1dfa22a8ecad
Normal-7958.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7958.png,Normal/images/Normal-7958.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7958.png,Normal/masks/Normal-7958.png,True,False,299,299,L,163.97,95a13b64b6794224df21231696715f01
Normal-7959.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7959.png,Normal/images/Normal-7959.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7959.png,Normal/masks/Normal-7959.png,True,False,299,299,L,140.34,a0a0abe2290b53c10c480ed0edc4b104
Normal-796.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-796.png,Normal/images/Normal-796.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-796.png,Normal/masks/Normal-796.png,True,False,299,299,L,130.24,61593a6db3c358bab43ccbcad5639cff
Normal-7960.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7960.png,Normal/images/Normal-7960.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7960.png,Normal/masks/Normal-7960.png,True,False,299,299,L,120.52,8f1292ec24ac3809f8432bd13478343c
Normal-7961.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7961.png,Normal/images/Normal-7961.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7961.png,Normal/masks/Normal-7961.png,True,False,299,299,L,133.94,bf64a43abc0bacf3f3780168fa1ce94f
Normal-7962.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7962.png,Normal/images/Normal-7962.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7962.png,Normal/masks/Normal-7962.png,True,False,299,299,L,110.49,7b913abe0b35e8ff059591b3b20437bf
Normal-7963.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7963.png,Normal/images/Normal-7963.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7963.png,Normal/masks/Normal-7963.png,True,False,299,299,L,109.28,fc743f365a195e3f924b2ae5992cb5c0
Normal-7964.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7964.png,Normal/images/Normal-7964.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7964.png,Normal/masks/Normal-7964.png,True,False,299,299,L,118.51,67efc9f2bab37799d83e820c1663ebc5
Normal-7965.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7965.png,Normal/images/Normal-7965.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7965.png,Normal/masks/Normal-7965.png,True,False,299,299,L,119.21,2f28e78a1c07463257c160e14927a7c9
Normal-7966.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7966.png,Normal/images/Normal-7966.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7966.png,Normal/masks/Normal-7966.png,True,False,299,299,L,118.86,32cd1dd7099d8c492e4b0cd40c537019
Normal-7967.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7967.png,Normal/images/Normal-7967.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7967.png,Normal/masks/Normal-7967.png,True,False,299,299,L,147.24,c76b8120646e23a51f8e46f033838c86
Normal-7968.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7968.png,Normal/images/Normal-7968.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7968.png,Normal/masks/Normal-7968.png,True,False,299,299,L,124.26,dc4f8188ec107dd5b472497a44c500e5
Normal-7969.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7969.png,Normal/images/Normal-7969.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7969.png,Normal/masks/Normal-7969.png,True,False,299,299,L,127.53,ec70b4881186a4436a123d57ea621dc2
Normal-797.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-797.png,Normal/images/Normal-797.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-797.png,Normal/masks/Normal-797.png,True,False,299,299,L,111.37,7ab3cfd7569bd0fa2f283c460ddd05ce
Normal-7970.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7970.png,Normal/images/Normal-7970.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7970.png,Normal/masks/Normal-7970.png,True,False,299,299,L,155.68,02a8270e149749ee9cfbd352ab760be5
Normal-7971.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7971.png,Normal/images/Normal-7971.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7971.png,Normal/masks/Normal-7971.png,True,False,299,299,L,94.1,f0f6460f1bfcde3830c8709661f3c58a
Normal-7972.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7972.png,Normal/images/Normal-7972.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7972.png,Normal/masks/Normal-7972.png,True,False,299,299,L,105.89,18bdb645b47c5ee3b705b70c94fcd6ea
Normal-7973.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7973.png,Normal/images/Normal-7973.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7973.png,Normal/masks/Normal-7973.png,True,False,299,299,L,136.2,853c87f9a99346b5bc0a1b85141dd368
Normal-7974.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7974.png,Normal/images/Normal-7974.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7974.png,Normal/masks/Normal-7974.png,True,False,299,299,L,137.36,52056355fc638ea85ef2f5839c3f2f7d
Normal-7975.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7975.png,Normal/images/Normal-7975.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7975.png,Normal/masks/Normal-7975.png,True,False,299,299,L,121.64,eeb2b85f21a4fa88b4aa199d89849f2c
Normal-7976.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7976.png,Normal/images/Normal-7976.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7976.png,Normal/masks/Normal-7976.png,True,False,299,299,L,128.75,99d119b09730b68a2393076fab6792e4
Normal-7977.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7977.png,Normal/images/Normal-7977.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7977.png,Normal/masks/Normal-7977.png,True,False,299,299,L,149.11,7103856f20881660d6a0b53682c67c2f
Normal-7978.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7978.png,Normal/images/Normal-7978.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7978.png,Normal/masks/Normal-7978.png,True,False,299,299,L,152.27,9feaaabb423a52f4b058bf52f86e1ecd
Normal-7979.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7979.png,Normal/images/Normal-7979.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7979.png,Normal/masks/Normal-7979.png,True,False,299,299,L,99.71,9a696c9e2c30a10a5985622fd9c5203d
Normal-798.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-798.png,Normal/images/Normal-798.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-798.png,Normal/masks/Normal-798.png,True,False,299,299,L,128.26,a28462fc9c07b4e64d5a7ae9584814fc
Normal-7980.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7980.png,Normal/images/Normal-7980.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7980.png,Normal/masks/Normal-7980.png,True,False,299,299,L,77.66,53b49d56f1aa616279e3cb2eaeee5a1a
Normal-7981.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7981.png,Normal/images/Normal-7981.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7981.png,Normal/masks/Normal-7981.png,True,False,299,299,L,107.87,f43098f48d40230a4e1ed405aa6fe865
Normal-7982.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7982.png,Normal/images/Normal-7982.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7982.png,Normal/masks/Normal-7982.png,True,False,299,299,L,135.01,a03c4f4cd9faacf4d2fb54f393538957
Normal-7983.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7983.png,Normal/images/Normal-7983.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7983.png,Normal/masks/Normal-7983.png,True,False,299,299,L,167.7,df785449dbd3a64418a65766e753e027
Normal-7984.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7984.png,Normal/images/Normal-7984.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7984.png,Normal/masks/Normal-7984.png,True,False,299,299,L,163.82,29bb8885bf0e35d4242e24b4039a0575
Normal-7985.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7985.png,Normal/images/Normal-7985.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7985.png,Normal/masks/Normal-7985.png,True,False,299,299,L,143.11,c90c404d40a5de97522ada1ccef9ee65
Normal-7986.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7986.png,Normal/images/Normal-7986.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7986.png,Normal/masks/Normal-7986.png,True,False,299,299,L,93.73,bf1a5f6f7a0e0309db2aad2d57c6eaf7
Normal-7987.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7987.png,Normal/images/Normal-7987.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7987.png,Normal/masks/Normal-7987.png,True,False,299,299,L,123.45,574e0a6fd917966db58cc17415c065f5
Normal-7988.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7988.png,Normal/images/Normal-7988.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7988.png,Normal/masks/Normal-7988.png,True,False,299,299,L,113.64,eecd5e4d98b75407b315f75b884e6de8
Normal-7989.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7989.png,Normal/images/Normal-7989.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7989.png,Normal/masks/Normal-7989.png,True,False,299,299,L,161.69,a7f6bce44fec12678fb3e603e6a18b0f
Normal-799.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-799.png,Normal/images/Normal-799.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-799.png,Normal/masks/Normal-799.png,True,False,299,299,L,113.33,8c1e45bc7383d00915a1979e7f88f612
Normal-7990.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7990.png,Normal/images/Normal-7990.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7990.png,Normal/masks/Normal-7990.png,True,False,299,299,L,150.92,acb4254be108f9e654e4ec9257f79418
Normal-7991.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7991.png,Normal/images/Normal-7991.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7991.png,Normal/masks/Normal-7991.png,True,False,299,299,L,129.04,19075178f411c069a69b9ef32faeba30
Normal-7992.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7992.png,Normal/images/Normal-7992.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7992.png,Normal/masks/Normal-7992.png,True,False,299,299,L,122.94,ff4bef088ab2b2f0d27132668bd3f7df
Normal-7993.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7993.png,Normal/images/Normal-7993.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7993.png,Normal/masks/Normal-7993.png,True,False,299,299,L,116.37,9322712b3554f8b6f7d612dc025395e0
Normal-7994.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7994.png,Normal/images/Normal-7994.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7994.png,Normal/masks/Normal-7994.png,True,False,299,299,L,108.77,25c217bca672d1f1b913a26b12a7d092
Normal-7995.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7995.png,Normal/images/Normal-7995.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7995.png,Normal/masks/Normal-7995.png,True,False,299,299,L,161.02,80d6e7b634e9e608ba8e00ae32063930
Normal-7996.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7996.png,Normal/images/Normal-7996.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7996.png,Normal/masks/Normal-7996.png,True,False,299,299,L,134.83,a91ea9b615f0175e3620b56286d01d6a
Normal-7997.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7997.png,Normal/images/Normal-7997.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7997.png,Normal/masks/Normal-7997.png,True,False,299,299,L,137.97,81df49f1ca313859663b880c1fae128a
Normal-7998.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7998.png,Normal/images/Normal-7998.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7998.png,Normal/masks/Normal-7998.png,True,False,299,299,L,115.43,eeab07b4ee1b26bc16b431c1d308ac31
Normal-7999.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-7999.png,Normal/images/Normal-7999.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-7999.png,Normal/masks/Normal-7999.png,True,False,299,299,L,141.16,322ccc48b9955166ebc7c9a25ae25d3a
Normal-8.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8.png,Normal/images/Normal-8.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8.png,Normal/masks/Normal-8.png,True,False,299,299,L,133.65,b0c39901dd2c6a4985ac723a7d9a0fae
Normal-80.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-80.png,Normal/images/Normal-80.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-80.png,Normal/masks/Normal-80.png,True,False,299,299,L,119.62,7b605554e8cdc566865d112585b573c7
Normal-800.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-800.png,Normal/images/Normal-800.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-800.png,Normal/masks/Normal-800.png,True,False,299,299,L,126.93,3709d3e9490c8c93eebc9fa8f84fd030
Normal-8000.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8000.png,Normal/images/Normal-8000.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8000.png,Normal/masks/Normal-8000.png,True,False,299,299,L,128.21,5cbe96f12ad6af3d7407856602fc6c45
Normal-8001.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8001.png,Normal/images/Normal-8001.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8001.png,Normal/masks/Normal-8001.png,True,False,299,299,L,160.52,03edc1d2722a3ef9d33d7d1d9decec11
Normal-8002.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8002.png,Normal/images/Normal-8002.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8002.png,Normal/masks/Normal-8002.png,True,False,299,299,L,165.77,16ff10c6188bd35c98813a6b6f7af4ab
Normal-8003.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8003.png,Normal/images/Normal-8003.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8003.png,Normal/masks/Normal-8003.png,True,False,299,299,L,102.26,d44d9cb757670ed9b16c16efa691f78d
Normal-8004.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8004.png,Normal/images/Normal-8004.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8004.png,Normal/masks/Normal-8004.png,True,False,299,299,L,157.39,b37aaab055c0a237bbdafc8a05e79b87
Normal-8005.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8005.png,Normal/images/Normal-8005.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8005.png,Normal/masks/Normal-8005.png,True,False,299,299,L,156.5,8c609f057e6f39f42e2962ebf494c7ee
Normal-8006.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8006.png,Normal/images/Normal-8006.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8006.png,Normal/masks/Normal-8006.png,True,False,299,299,L,119.38,4b909f0483e7e1971537d8d1fba5da45
Normal-8007.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8007.png,Normal/images/Normal-8007.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8007.png,Normal/masks/Normal-8007.png,True,False,299,299,L,127.94,c1d78ea3e658360c09751aa04b8c685c
Normal-8008.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8008.png,Normal/images/Normal-8008.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8008.png,Normal/masks/Normal-8008.png,True,False,299,299,L,125.62,2bd7e897c3674ef5c3ae462bdc17216e
Normal-8009.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8009.png,Normal/images/Normal-8009.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8009.png,Normal/masks/Normal-8009.png,True,False,299,299,L,107.41,1b47d1bf2e9718afccf4c4a51d7c08aa
Normal-801.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-801.png,Normal/images/Normal-801.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-801.png,Normal/masks/Normal-801.png,True,False,299,299,L,106.16,d80c6ef7877555315911345dd88c273e
Normal-8010.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8010.png,Normal/images/Normal-8010.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8010.png,Normal/masks/Normal-8010.png,True,False,299,299,L,126.68,6674d78fb729b10f67c8dbe1123a378b
Normal-8011.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8011.png,Normal/images/Normal-8011.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8011.png,Normal/masks/Normal-8011.png,True,False,299,299,L,142.3,eec8ee4fe8e1a491aac8eb08da135026
Normal-8012.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8012.png,Normal/images/Normal-8012.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8012.png,Normal/masks/Normal-8012.png,True,False,299,299,L,119.69,9b17d1286609bb473eb5845f05b74d95
Normal-8013.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8013.png,Normal/images/Normal-8013.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8013.png,Normal/masks/Normal-8013.png,True,False,299,299,L,111.76,8fcb14dabd1a162d98713bad31c81e0c
Normal-8014.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8014.png,Normal/images/Normal-8014.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8014.png,Normal/masks/Normal-8014.png,True,False,299,299,L,113.92,c817df36182bdb0f8b944bad0c5f7784
Normal-8015.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8015.png,Normal/images/Normal-8015.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8015.png,Normal/masks/Normal-8015.png,True,False,299,299,L,103.8,d014668c7ce1b328e966e26063b9156b
Normal-8016.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8016.png,Normal/images/Normal-8016.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8016.png,Normal/masks/Normal-8016.png,True,False,299,299,L,154.68,f761754988f73a0119bb1ea6166c1e17
Normal-8017.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8017.png,Normal/images/Normal-8017.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8017.png,Normal/masks/Normal-8017.png,True,False,299,299,L,126.17,598eb64e1240bf709964a2e6171ef14d
Normal-8018.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8018.png,Normal/images/Normal-8018.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8018.png,Normal/masks/Normal-8018.png,True,False,299,299,L,137.37,9c06cb7063c0006f2a360fbb6ae59eb5
Normal-8019.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8019.png,Normal/images/Normal-8019.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8019.png,Normal/masks/Normal-8019.png,True,False,299,299,L,155.88,c67867248417124ef099758a314e2c9c
Normal-802.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-802.png,Normal/images/Normal-802.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-802.png,Normal/masks/Normal-802.png,True,False,299,299,L,125.44,a8b8f4a75681d3361255164696394d09
Normal-8020.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8020.png,Normal/images/Normal-8020.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8020.png,Normal/masks/Normal-8020.png,True,False,299,299,L,141.46,9392ebcb8c6f03e8414cd2d480929b3a
Normal-8021.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8021.png,Normal/images/Normal-8021.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8021.png,Normal/masks/Normal-8021.png,True,False,299,299,L,100.09,f7436b38e4d2ac4976ef427ff3a06ebe
Normal-8022.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8022.png,Normal/images/Normal-8022.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8022.png,Normal/masks/Normal-8022.png,True,False,299,299,L,120.12,9896495bab7bae3b7bad73481722e051
Normal-8023.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8023.png,Normal/images/Normal-8023.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8023.png,Normal/masks/Normal-8023.png,True,False,299,299,L,97.68,c3a5d21e5304c8dde32a16c1f464aa9a
Normal-8024.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8024.png,Normal/images/Normal-8024.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8024.png,Normal/masks/Normal-8024.png,True,False,299,299,L,114.3,cc4fed3b5c70431de7a69e09552c0f50
Normal-8025.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8025.png,Normal/images/Normal-8025.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8025.png,Normal/masks/Normal-8025.png,True,False,299,299,L,122.53,a3397e18da5e47f184ff5a1cf06e9085
Normal-8026.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8026.png,Normal/images/Normal-8026.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8026.png,Normal/masks/Normal-8026.png,True,False,299,299,L,123.75,75909f3d7d0d265238774997c2219eef
Normal-8027.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8027.png,Normal/images/Normal-8027.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8027.png,Normal/masks/Normal-8027.png,True,False,299,299,L,95.2,25b92a0c2daacf8ac77c6d4817333688
Normal-8028.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8028.png,Normal/images/Normal-8028.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8028.png,Normal/masks/Normal-8028.png,True,False,299,299,L,122.22,27047f3146d9eae13568b1d1a4553847
Normal-8029.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8029.png,Normal/images/Normal-8029.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8029.png,Normal/masks/Normal-8029.png,True,False,299,299,L,119.6,57dd91afb539a6bc1b69865b4abff0ee
Normal-803.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-803.png,Normal/images/Normal-803.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-803.png,Normal/masks/Normal-803.png,True,False,299,299,L,132.06,bf03c67444c51ae98548161e032df60f
Normal-8030.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8030.png,Normal/images/Normal-8030.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8030.png,Normal/masks/Normal-8030.png,True,False,299,299,L,152.89,25b11e445fb8542896ded58effb388ec
Normal-8031.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8031.png,Normal/images/Normal-8031.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8031.png,Normal/masks/Normal-8031.png,True,False,299,299,L,162.63,2e2b3a6e0a594286c282f584d5659e89
Normal-8032.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8032.png,Normal/images/Normal-8032.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8032.png,Normal/masks/Normal-8032.png,True,False,299,299,L,117.73,df281512d00e9c423a8a4e7abe7c71b7
Normal-8033.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8033.png,Normal/images/Normal-8033.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8033.png,Normal/masks/Normal-8033.png,True,False,299,299,L,162.07,e2c1c75f7a2aad0fb8c60f4fd02b58aa
Normal-8034.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8034.png,Normal/images/Normal-8034.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8034.png,Normal/masks/Normal-8034.png,True,False,299,299,L,162.75,a833abf4fa5c32ec1bce369e7d074803
Normal-8035.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8035.png,Normal/images/Normal-8035.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8035.png,Normal/masks/Normal-8035.png,True,False,299,299,L,92.45,1bff4d794e9bcec8163a1c9b225e8fa5
Normal-8036.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8036.png,Normal/images/Normal-8036.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8036.png,Normal/masks/Normal-8036.png,True,False,299,299,L,116.53,62f7c910b52463f37bdb8b9a610d77fc
Normal-8037.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8037.png,Normal/images/Normal-8037.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8037.png,Normal/masks/Normal-8037.png,True,False,299,299,L,149.09,08e1dd06c5f6995648fee940415ae86f
Normal-8038.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8038.png,Normal/images/Normal-8038.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8038.png,Normal/masks/Normal-8038.png,True,False,299,299,L,149.66,7cc59786c8b2af3944e93386950059f3
Normal-8039.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8039.png,Normal/images/Normal-8039.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8039.png,Normal/masks/Normal-8039.png,True,False,299,299,L,107.45,0ea03332b7172e26a9c32ba39d1716b2
Normal-804.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-804.png,Normal/images/Normal-804.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-804.png,Normal/masks/Normal-804.png,True,False,299,299,L,122.39,d32b538214df04593143d9c00ee14053
Normal-8040.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8040.png,Normal/images/Normal-8040.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8040.png,Normal/masks/Normal-8040.png,True,False,299,299,L,106.4,3fd5d66c2bfb3fb80d35bee3e45e17ca
Normal-8041.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8041.png,Normal/images/Normal-8041.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8041.png,Normal/masks/Normal-8041.png,True,False,299,299,L,132.17,cc29cf7ada13502a44c76dcf93d19399
Normal-8042.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8042.png,Normal/images/Normal-8042.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8042.png,Normal/masks/Normal-8042.png,True,False,299,299,L,162.69,7b55d99c2e4941336472ecc7df951893
Normal-8043.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8043.png,Normal/images/Normal-8043.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8043.png,Normal/masks/Normal-8043.png,True,False,299,299,L,169.95,c4131df2b3de3412844502534676f6c8
Normal-8044.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8044.png,Normal/images/Normal-8044.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8044.png,Normal/masks/Normal-8044.png,True,False,299,299,L,111.07,14d6b91f499f2440372e45562fa76dbc
Normal-8045.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8045.png,Normal/images/Normal-8045.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8045.png,Normal/masks/Normal-8045.png,True,False,299,299,L,170.38,c0cadd25a58461885bae287a46712e3d
Normal-8046.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8046.png,Normal/images/Normal-8046.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8046.png,Normal/masks/Normal-8046.png,True,False,299,299,L,113.93,728855d6acbd50033c8e24b9716b4786
Normal-8047.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8047.png,Normal/images/Normal-8047.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8047.png,Normal/masks/Normal-8047.png,True,False,299,299,L,156.93,6ea4937940aa2a4e8266e27da1e52c1f
Normal-8048.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8048.png,Normal/images/Normal-8048.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8048.png,Normal/masks/Normal-8048.png,True,False,299,299,L,112.94,fb1bdeb3f4f32b24a512718cfc467a4e
Normal-8049.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8049.png,Normal/images/Normal-8049.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8049.png,Normal/masks/Normal-8049.png,True,False,299,299,L,106.16,3bc2568f8aa59062ccb956157867fca2
Normal-805.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-805.png,Normal/images/Normal-805.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-805.png,Normal/masks/Normal-805.png,True,False,299,299,L,136.85,48ee98adef1c363933271a1fd5ef0e4e
Normal-8050.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8050.png,Normal/images/Normal-8050.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8050.png,Normal/masks/Normal-8050.png,True,False,299,299,L,156.75,0b53ea81d658a36e7cdc4e67cfba6aea
Normal-8051.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8051.png,Normal/images/Normal-8051.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8051.png,Normal/masks/Normal-8051.png,True,False,299,299,L,105.68,baae4c0758ed4688b15332d9a1e6eb24
Normal-8052.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8052.png,Normal/images/Normal-8052.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8052.png,Normal/masks/Normal-8052.png,True,False,299,299,L,165.89,1a3527755d72a7ba7ccae3fb8159c86c
Normal-8053.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8053.png,Normal/images/Normal-8053.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8053.png,Normal/masks/Normal-8053.png,True,False,299,299,L,89.44,aef879b278d0ced33e43f5ab1fcaf443
Normal-8054.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8054.png,Normal/images/Normal-8054.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8054.png,Normal/masks/Normal-8054.png,True,False,299,299,L,137.8,495fe5b850cfd0898756731e56ac3891
Normal-8055.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8055.png,Normal/images/Normal-8055.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8055.png,Normal/masks/Normal-8055.png,True,False,299,299,L,149.22,cdd109921e8c3dae7f2516636d18d5ca
Normal-8056.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8056.png,Normal/images/Normal-8056.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8056.png,Normal/masks/Normal-8056.png,True,False,299,299,L,64.35,1d92a1a2228e8f1ba4dfc0a7ec0b6448
Normal-8057.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8057.png,Normal/images/Normal-8057.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8057.png,Normal/masks/Normal-8057.png,True,False,299,299,L,152.84,fca0949310657b64b5489ab951caf51b
Normal-8058.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8058.png,Normal/images/Normal-8058.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8058.png,Normal/masks/Normal-8058.png,True,False,299,299,L,116.86,4e1fdebdd08cac80b73e4cfb21029cda
Normal-8059.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8059.png,Normal/images/Normal-8059.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8059.png,Normal/masks/Normal-8059.png,True,False,299,299,L,130.29,6c9bd0fc1d32b998778eb86e5bb5d75d
Normal-806.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-806.png,Normal/images/Normal-806.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-806.png,Normal/masks/Normal-806.png,True,False,299,299,L,134.02,428080c4583581bc2e8b9640c9f582c2
Normal-8060.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8060.png,Normal/images/Normal-8060.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8060.png,Normal/masks/Normal-8060.png,True,False,299,299,L,145.28,2852a92024eeb67f82b0bc5ad5692679
Normal-8061.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8061.png,Normal/images/Normal-8061.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8061.png,Normal/masks/Normal-8061.png,True,False,299,299,L,127.97,a8e3b27f176d89f3594c1251521d7d4b
Normal-8062.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8062.png,Normal/images/Normal-8062.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8062.png,Normal/masks/Normal-8062.png,True,False,299,299,L,101.14,241c0ea693ecef13438d6771efe58c5c
Normal-8063.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8063.png,Normal/images/Normal-8063.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8063.png,Normal/masks/Normal-8063.png,True,False,299,299,L,157.05,fe23534825cb214908a8ce6327418901
Normal-8064.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8064.png,Normal/images/Normal-8064.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8064.png,Normal/masks/Normal-8064.png,True,False,299,299,L,158.74,e3219259433f9042847da80585461bd1
Normal-8065.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8065.png,Normal/images/Normal-8065.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8065.png,Normal/masks/Normal-8065.png,True,False,299,299,L,175.4,12666ffd34d2a25b61b119367f9c3344
Normal-8066.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8066.png,Normal/images/Normal-8066.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8066.png,Normal/masks/Normal-8066.png,True,False,299,299,L,118.82,189d9c069499e2886a91496c77c0d8ff
Normal-8067.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8067.png,Normal/images/Normal-8067.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8067.png,Normal/masks/Normal-8067.png,True,False,299,299,L,138.68,e86dc0cc229aede85d60228ca3d29eea
Normal-8068.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8068.png,Normal/images/Normal-8068.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8068.png,Normal/masks/Normal-8068.png,True,False,299,299,L,91.48,ff5f424c5f243b3298bf393bfaaa8b41
Normal-8069.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8069.png,Normal/images/Normal-8069.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8069.png,Normal/masks/Normal-8069.png,True,False,299,299,L,115.67,8a26355ea29c0ab36eeb18ac497f43bc
Normal-807.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-807.png,Normal/images/Normal-807.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-807.png,Normal/masks/Normal-807.png,True,False,299,299,L,138.15,324ed55e49b1d5fec7ab2abb8c14e418
Normal-8070.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8070.png,Normal/images/Normal-8070.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8070.png,Normal/masks/Normal-8070.png,True,False,299,299,L,100.43,4942a144e5a7ba8ef8eb612e49a7d4cf
Normal-8071.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8071.png,Normal/images/Normal-8071.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8071.png,Normal/masks/Normal-8071.png,True,False,299,299,L,124.15,9aed9a12bbd9714f9cd2eb65fe3ab9a9
Normal-8072.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8072.png,Normal/images/Normal-8072.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8072.png,Normal/masks/Normal-8072.png,True,False,299,299,L,158.54,8390d4e568b1ab5e1195a501dfc0f3db
Normal-8073.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8073.png,Normal/images/Normal-8073.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8073.png,Normal/masks/Normal-8073.png,True,False,299,299,L,121.16,d54d6e4afd7879fe921992e01c4980d2
Normal-8074.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8074.png,Normal/images/Normal-8074.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8074.png,Normal/masks/Normal-8074.png,True,False,299,299,L,117.37,118f6c4cb32a6ab5bedeb6ca96249a85
Normal-8075.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8075.png,Normal/images/Normal-8075.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8075.png,Normal/masks/Normal-8075.png,True,False,299,299,L,123.39,316fd6027b5d5cea2243184f39f7726c
Normal-8076.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8076.png,Normal/images/Normal-8076.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8076.png,Normal/masks/Normal-8076.png,True,False,299,299,L,136.66,7d632636b933bbd8ebb91be1796a9525
Normal-8077.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8077.png,Normal/images/Normal-8077.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8077.png,Normal/masks/Normal-8077.png,True,False,299,299,L,108.68,c70bd4f6016ba38ede981035f2b1cad7
Normal-8078.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8078.png,Normal/images/Normal-8078.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8078.png,Normal/masks/Normal-8078.png,True,False,299,299,L,141.25,b9b21b436788f13dacfd166f213f30cc
Normal-8079.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8079.png,Normal/images/Normal-8079.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8079.png,Normal/masks/Normal-8079.png,True,False,299,299,L,160.09,5a74ee239d33ea4d192f00a0eef5ac23
Normal-808.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-808.png,Normal/images/Normal-808.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-808.png,Normal/masks/Normal-808.png,True,False,299,299,L,125.79,f3a85b118912d044d1bc8ed292a51a57
Normal-8080.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8080.png,Normal/images/Normal-8080.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8080.png,Normal/masks/Normal-8080.png,True,False,299,299,L,120.12,ed1ca4e2c51ceea3c59c438511822eda
Normal-8081.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8081.png,Normal/images/Normal-8081.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8081.png,Normal/masks/Normal-8081.png,True,False,299,299,L,110.14,6077e951552800796dd23ddfb4709e5f
Normal-8082.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8082.png,Normal/images/Normal-8082.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8082.png,Normal/masks/Normal-8082.png,True,False,299,299,L,106.87,1c268c33af7f98cbf4b5400682ac9a85
Normal-8083.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8083.png,Normal/images/Normal-8083.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8083.png,Normal/masks/Normal-8083.png,True,False,299,299,L,155.77,37daa01a162ba29b7c267f036002276f
Normal-8084.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8084.png,Normal/images/Normal-8084.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8084.png,Normal/masks/Normal-8084.png,True,False,299,299,L,158.52,11491a98a3331d4cb4e61e9ce355aa8b
Normal-8085.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8085.png,Normal/images/Normal-8085.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8085.png,Normal/masks/Normal-8085.png,True,False,299,299,L,168.26,6f3953b36097f9b952d13158ad10a9ac
Normal-8086.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8086.png,Normal/images/Normal-8086.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8086.png,Normal/masks/Normal-8086.png,True,False,299,299,L,98.57,a882ee8e3911270db424615327d2cb04
Normal-8087.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8087.png,Normal/images/Normal-8087.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8087.png,Normal/masks/Normal-8087.png,True,False,299,299,L,139.95,a54297370fa5f075a691225021794a17
Normal-8088.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8088.png,Normal/images/Normal-8088.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8088.png,Normal/masks/Normal-8088.png,True,False,299,299,L,136.38,5bdd8350cc8cbdd3f121e26165264f11
Normal-8089.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8089.png,Normal/images/Normal-8089.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8089.png,Normal/masks/Normal-8089.png,True,False,299,299,L,140.34,8914d868fb1558941a9c4cf17aa44974
Normal-809.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-809.png,Normal/images/Normal-809.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-809.png,Normal/masks/Normal-809.png,True,False,299,299,L,125.29,e8cedf0b551445ce13a51a407ebadfa6
Normal-8090.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8090.png,Normal/images/Normal-8090.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8090.png,Normal/masks/Normal-8090.png,True,False,299,299,L,96.16,9afda754b2aae58260f80b0ff69917f2
Normal-8091.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8091.png,Normal/images/Normal-8091.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8091.png,Normal/masks/Normal-8091.png,True,False,299,299,L,105.08,b0e272f3f64cb52945a0f36da2d86ac8
Normal-8092.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8092.png,Normal/images/Normal-8092.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8092.png,Normal/masks/Normal-8092.png,True,False,299,299,L,124.88,b4698ee9aa52e205a0ae1dceb5dcee20
Normal-8093.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8093.png,Normal/images/Normal-8093.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8093.png,Normal/masks/Normal-8093.png,True,False,299,299,L,139.7,00d9680ad369adce8c67432004b9564a
Normal-8094.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8094.png,Normal/images/Normal-8094.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8094.png,Normal/masks/Normal-8094.png,True,False,299,299,L,125.8,a01e690e82477b2846b637667a4fe619
Normal-8095.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8095.png,Normal/images/Normal-8095.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8095.png,Normal/masks/Normal-8095.png,True,False,299,299,L,162.41,afa8b9068f41d94c9e1e3510e260a45a
Normal-8096.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8096.png,Normal/images/Normal-8096.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8096.png,Normal/masks/Normal-8096.png,True,False,299,299,L,106.95,73e1933cfbf121542dac782f678524c2
Normal-8097.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8097.png,Normal/images/Normal-8097.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8097.png,Normal/masks/Normal-8097.png,True,False,299,299,L,166.8,9c85501ac8a8d0d81dc9db32e2e5a3ea
Normal-8098.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8098.png,Normal/images/Normal-8098.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8098.png,Normal/masks/Normal-8098.png,True,False,299,299,L,106.73,3587087409a908f49005f03745111e34
Normal-8099.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8099.png,Normal/images/Normal-8099.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8099.png,Normal/masks/Normal-8099.png,True,False,299,299,L,104.93,ad69f7a2a05ab4ea6c4dd552cd68df40
Normal-81.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-81.png,Normal/images/Normal-81.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-81.png,Normal/masks/Normal-81.png,True,False,299,299,L,101.82,d8efa4b8f8868e83c114ffcde6e5fecd
Normal-810.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-810.png,Normal/images/Normal-810.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-810.png,Normal/masks/Normal-810.png,True,False,299,299,L,122.04,60b63f74575e19e605f7af944108bece
Normal-8100.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8100.png,Normal/images/Normal-8100.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8100.png,Normal/masks/Normal-8100.png,True,False,299,299,L,116.89,cf3f3b3361ca62c375931131204bdf3b
Normal-8101.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8101.png,Normal/images/Normal-8101.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8101.png,Normal/masks/Normal-8101.png,True,False,299,299,L,180.63,6e0714d73564d811eba4fbca94104469
Normal-8102.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8102.png,Normal/images/Normal-8102.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8102.png,Normal/masks/Normal-8102.png,True,False,299,299,L,143.74,819e6d4dfc8fa8fa2c856468a2cd83c8
Normal-8103.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8103.png,Normal/images/Normal-8103.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8103.png,Normal/masks/Normal-8103.png,True,False,299,299,L,134.15,fb03ead140b68017b306f6c1a70c5d07
Normal-8104.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8104.png,Normal/images/Normal-8104.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8104.png,Normal/masks/Normal-8104.png,True,False,299,299,L,130.18,95d1e6460c3efa7a8efd4961cd2f8090
Normal-8105.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8105.png,Normal/images/Normal-8105.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8105.png,Normal/masks/Normal-8105.png,True,False,299,299,L,99.56,7290268685dffe63b0452830b03c361c
Normal-8106.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8106.png,Normal/images/Normal-8106.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8106.png,Normal/masks/Normal-8106.png,True,False,299,299,L,115.34,60d8aa02fdd9f890ff2e762a410b348c
Normal-8107.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8107.png,Normal/images/Normal-8107.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8107.png,Normal/masks/Normal-8107.png,True,False,299,299,L,161.81,5a9d413b4faaa300b89431cdd6e8cc0e
Normal-8108.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8108.png,Normal/images/Normal-8108.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8108.png,Normal/masks/Normal-8108.png,True,False,299,299,L,118.0,14b8fb933d730460961b481bc252f732
Normal-8109.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8109.png,Normal/images/Normal-8109.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8109.png,Normal/masks/Normal-8109.png,True,False,299,299,L,141.89,4efd7c8a30ea8847e6bdccbc1f4bc958
Normal-811.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-811.png,Normal/images/Normal-811.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-811.png,Normal/masks/Normal-811.png,True,False,299,299,L,125.7,c7990fc93337ec6e152a54cb8fcdc948
Normal-8110.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8110.png,Normal/images/Normal-8110.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8110.png,Normal/masks/Normal-8110.png,True,False,299,299,L,103.92,cfcea7aca01bcfda2017803602b35e6f
Normal-8111.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8111.png,Normal/images/Normal-8111.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8111.png,Normal/masks/Normal-8111.png,True,False,299,299,L,111.34,17b79a1f624fc5504491bfd156a62c8c
Normal-8112.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8112.png,Normal/images/Normal-8112.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8112.png,Normal/masks/Normal-8112.png,True,False,299,299,L,98.21,9d1e29f761e8bde4ef36619cc52b86b5
Normal-8113.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8113.png,Normal/images/Normal-8113.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8113.png,Normal/masks/Normal-8113.png,True,False,299,299,L,134.11,6221be26580b99059a93538de73f8b13
Normal-8114.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8114.png,Normal/images/Normal-8114.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8114.png,Normal/masks/Normal-8114.png,True,False,299,299,L,104.09,6d4c4c3a3568518e1fcf71258b18cdca
Normal-8115.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8115.png,Normal/images/Normal-8115.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8115.png,Normal/masks/Normal-8115.png,True,False,299,299,L,133.26,6d3b510164cb9c909a50a47440bac103
Normal-8116.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8116.png,Normal/images/Normal-8116.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8116.png,Normal/masks/Normal-8116.png,True,False,299,299,L,124.86,155f315325b91b023a3ec9b5905d5dff
Normal-8117.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8117.png,Normal/images/Normal-8117.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8117.png,Normal/masks/Normal-8117.png,True,False,299,299,L,106.46,c69bd9cba2804b75ff0840eb0773c5ca
Normal-8118.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8118.png,Normal/images/Normal-8118.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8118.png,Normal/masks/Normal-8118.png,True,False,299,299,L,150.33,f007bb65986aab0129619aa491a85d1b
Normal-8119.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8119.png,Normal/images/Normal-8119.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8119.png,Normal/masks/Normal-8119.png,True,False,299,299,L,151.13,2e1654264fba0d2842ca3d6501d504a3
Normal-812.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-812.png,Normal/images/Normal-812.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-812.png,Normal/masks/Normal-812.png,True,False,299,299,L,130.82,8bdc121b7b8ce5f956f6378538f2cb48
Normal-8120.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8120.png,Normal/images/Normal-8120.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8120.png,Normal/masks/Normal-8120.png,True,False,299,299,L,110.46,d8715620d3c707b8b975b61709f73c0a
Normal-8121.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8121.png,Normal/images/Normal-8121.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8121.png,Normal/masks/Normal-8121.png,True,False,299,299,L,136.71,530fe396b0da4ad195919c8d1f8f871c
Normal-8122.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8122.png,Normal/images/Normal-8122.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8122.png,Normal/masks/Normal-8122.png,True,False,299,299,L,104.86,b3ba42064990d744392dfb80c9b35774
Normal-8123.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8123.png,Normal/images/Normal-8123.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8123.png,Normal/masks/Normal-8123.png,True,False,299,299,L,111.45,3e6f1505fbb8a964b562796018c0c45e
Normal-8124.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8124.png,Normal/images/Normal-8124.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8124.png,Normal/masks/Normal-8124.png,True,False,299,299,L,139.39,4317f22b12d868bbe640322d84670442
Normal-8125.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8125.png,Normal/images/Normal-8125.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8125.png,Normal/masks/Normal-8125.png,True,False,299,299,L,145.04,fc96527d723b00ad135b09911244154e
Normal-8126.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8126.png,Normal/images/Normal-8126.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8126.png,Normal/masks/Normal-8126.png,True,False,299,299,L,115.16,8ba4ea7c16b870835a84a22ae07e00c8
Normal-8127.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8127.png,Normal/images/Normal-8127.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8127.png,Normal/masks/Normal-8127.png,True,False,299,299,L,162.55,152615c325f82742d6ea4d3a27e07d20
Normal-8128.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8128.png,Normal/images/Normal-8128.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8128.png,Normal/masks/Normal-8128.png,True,False,299,299,L,111.98,dd44c2ea5faa73a56a5f4a7dbd0b9101
Normal-8129.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8129.png,Normal/images/Normal-8129.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8129.png,Normal/masks/Normal-8129.png,True,False,299,299,L,132.03,fbda8c76016c3a43da0d47e8c6f5f77d
Normal-813.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-813.png,Normal/images/Normal-813.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-813.png,Normal/masks/Normal-813.png,True,False,299,299,L,117.04,44172caeeeb6d7bd3258bb58d5126f91
Normal-8130.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8130.png,Normal/images/Normal-8130.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8130.png,Normal/masks/Normal-8130.png,True,False,299,299,L,113.55,72628fb7bffa058747cdb2f45aa3f602
Normal-8131.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8131.png,Normal/images/Normal-8131.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8131.png,Normal/masks/Normal-8131.png,True,False,299,299,L,163.34,0e1ef0038b746e63acf891a3b986d6c6
Normal-8132.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8132.png,Normal/images/Normal-8132.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8132.png,Normal/masks/Normal-8132.png,True,False,299,299,L,107.11,b9a50231e7e41d66fc6e126d5534c535
Normal-8133.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8133.png,Normal/images/Normal-8133.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8133.png,Normal/masks/Normal-8133.png,True,False,299,299,L,139.76,3f9279d8bc10033fcd4203b5a9c54835
Normal-8134.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8134.png,Normal/images/Normal-8134.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8134.png,Normal/masks/Normal-8134.png,True,False,299,299,L,110.28,5d134197a84d96b2eb5a4d8c5b08a615
Normal-8135.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8135.png,Normal/images/Normal-8135.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8135.png,Normal/masks/Normal-8135.png,True,False,299,299,L,157.27,26473f0a80d204a4cd3e0d55ff74d5d5
Normal-8136.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8136.png,Normal/images/Normal-8136.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8136.png,Normal/masks/Normal-8136.png,True,False,299,299,L,121.73,a8f8f166f66552a4f712fcfa59ef44fa
Normal-8137.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8137.png,Normal/images/Normal-8137.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8137.png,Normal/masks/Normal-8137.png,True,False,299,299,L,133.86,999fbc217c17ab4dcd42ea1c67128830
Normal-8138.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8138.png,Normal/images/Normal-8138.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8138.png,Normal/masks/Normal-8138.png,True,False,299,299,L,171.1,e440c8259baf59fc655012e645d477a5
Normal-8139.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8139.png,Normal/images/Normal-8139.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8139.png,Normal/masks/Normal-8139.png,True,False,299,299,L,165.66,9618d69126db199cf013d051ae9a0105
Normal-814.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-814.png,Normal/images/Normal-814.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-814.png,Normal/masks/Normal-814.png,True,False,299,299,L,126.8,8308ae4c89b50001112ae8e10020d18a
Normal-8140.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8140.png,Normal/images/Normal-8140.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8140.png,Normal/masks/Normal-8140.png,True,False,299,299,L,160.0,c520681797d82a50b21c5c0da363a41d
Normal-8141.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8141.png,Normal/images/Normal-8141.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8141.png,Normal/masks/Normal-8141.png,True,False,299,299,L,131.63,836c9c67dd371d20c8dd7e6b399cc375
Normal-8142.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8142.png,Normal/images/Normal-8142.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8142.png,Normal/masks/Normal-8142.png,True,False,299,299,L,132.09,9a126ce7f64aeee262fad9b57b8878ad
Normal-8143.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8143.png,Normal/images/Normal-8143.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8143.png,Normal/masks/Normal-8143.png,True,False,299,299,L,115.61,9ad4e71256721f5135ddb592bb1178e7
Normal-8144.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8144.png,Normal/images/Normal-8144.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8144.png,Normal/masks/Normal-8144.png,True,False,299,299,L,101.18,7e5e11113e5193dacefb92810919b29c
Normal-8145.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8145.png,Normal/images/Normal-8145.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8145.png,Normal/masks/Normal-8145.png,True,False,299,299,L,122.27,ea93cf9562c8ffa6c2770d2c5baa5138
Normal-8146.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8146.png,Normal/images/Normal-8146.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8146.png,Normal/masks/Normal-8146.png,True,False,299,299,L,101.79,f6a33733e637bdf35d5879b77de26c35
Normal-8147.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8147.png,Normal/images/Normal-8147.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8147.png,Normal/masks/Normal-8147.png,True,False,299,299,L,147.51,c82f1a55bccb8418fc3639da3afa2849
Normal-8148.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8148.png,Normal/images/Normal-8148.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8148.png,Normal/masks/Normal-8148.png,True,False,299,299,L,66.68,98f1ba336580b26ad4893138c8a44e1b
Normal-8149.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8149.png,Normal/images/Normal-8149.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8149.png,Normal/masks/Normal-8149.png,True,False,299,299,L,116.63,abadc9370fce6028bbf015f1c61cca87
Normal-815.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-815.png,Normal/images/Normal-815.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-815.png,Normal/masks/Normal-815.png,True,False,299,299,L,93.65,f2d5445ebe19977d61f6f5c45e1c1c4a
Normal-8150.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8150.png,Normal/images/Normal-8150.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8150.png,Normal/masks/Normal-8150.png,True,False,299,299,L,130.09,fc7cce9ad4d8b95eec39629e98c306c4
Normal-8151.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8151.png,Normal/images/Normal-8151.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8151.png,Normal/masks/Normal-8151.png,True,False,299,299,L,138.56,9f99edd480964fd3c6df1209629702e6
Normal-8152.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8152.png,Normal/images/Normal-8152.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8152.png,Normal/masks/Normal-8152.png,True,False,299,299,L,122.91,69766c4aef41ea8a3e2f75391ca10737
Normal-8153.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8153.png,Normal/images/Normal-8153.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8153.png,Normal/masks/Normal-8153.png,True,False,299,299,L,113.86,d59857db16df1986ba67339a3a61d6ad
Normal-8154.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8154.png,Normal/images/Normal-8154.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8154.png,Normal/masks/Normal-8154.png,True,False,299,299,L,129.26,69a1a113ed0a88c387ee74fd8a1a58f8
Normal-8155.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8155.png,Normal/images/Normal-8155.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8155.png,Normal/masks/Normal-8155.png,True,False,299,299,L,117.26,2377877f13a753b0ce10da23a41b4b41
Normal-8156.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8156.png,Normal/images/Normal-8156.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8156.png,Normal/masks/Normal-8156.png,True,False,299,299,L,140.01,115697ffb0b8a205f5769e7977c1978c
Normal-8157.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8157.png,Normal/images/Normal-8157.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8157.png,Normal/masks/Normal-8157.png,True,False,299,299,L,103.7,675fa9cfd46a66f896cd26bd6e605221
Normal-8158.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8158.png,Normal/images/Normal-8158.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8158.png,Normal/masks/Normal-8158.png,True,False,299,299,L,123.21,a1dd1c53d2b0ed5e2547e0383851856e
Normal-8159.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8159.png,Normal/images/Normal-8159.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8159.png,Normal/masks/Normal-8159.png,True,False,299,299,L,106.29,c6aacf954117846fea594866d187e036
Normal-816.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-816.png,Normal/images/Normal-816.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-816.png,Normal/masks/Normal-816.png,True,False,299,299,L,130.82,94c16c5cd88e09918ea18b2623afc89b
Normal-8160.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8160.png,Normal/images/Normal-8160.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8160.png,Normal/masks/Normal-8160.png,True,False,299,299,L,140.92,cc1fefcc741fad197e8ae407e38b6068
Normal-8161.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8161.png,Normal/images/Normal-8161.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8161.png,Normal/masks/Normal-8161.png,True,False,299,299,L,131.88,bf203599aaf49cfb79ed390b2afbb51e
Normal-8162.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8162.png,Normal/images/Normal-8162.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8162.png,Normal/masks/Normal-8162.png,True,False,299,299,L,130.8,76d4c2071e1950996cbf10d4f91aa961
Normal-8163.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8163.png,Normal/images/Normal-8163.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8163.png,Normal/masks/Normal-8163.png,True,False,299,299,L,101.96,220db424c44dcb5684237cf555cc6339
Normal-8164.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8164.png,Normal/images/Normal-8164.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8164.png,Normal/masks/Normal-8164.png,True,False,299,299,L,156.7,05c43d06af8b9c39c4ce68167cccacd6
Normal-8165.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8165.png,Normal/images/Normal-8165.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8165.png,Normal/masks/Normal-8165.png,True,False,299,299,L,151.94,94a7742f3a5f0b57317aa84d3fe02555
Normal-8166.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8166.png,Normal/images/Normal-8166.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8166.png,Normal/masks/Normal-8166.png,True,False,299,299,L,85.8,61901a867421c844e2051fd2bb99e637
Normal-8167.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8167.png,Normal/images/Normal-8167.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8167.png,Normal/masks/Normal-8167.png,True,False,299,299,L,144.57,903030f9c4b0d40952eac70de74cc545
Normal-8168.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8168.png,Normal/images/Normal-8168.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8168.png,Normal/masks/Normal-8168.png,True,False,299,299,L,123.77,4893e6c4173bd9f740e8d5409994ce9a
Normal-8169.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8169.png,Normal/images/Normal-8169.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8169.png,Normal/masks/Normal-8169.png,True,False,299,299,L,103.76,e7996a0b0d5f3f33e8405ea33b990c5f
Normal-817.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-817.png,Normal/images/Normal-817.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-817.png,Normal/masks/Normal-817.png,True,False,299,299,L,164.0,afe5ddfc8aa54b115eedd6c5d18ea439
Normal-8170.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8170.png,Normal/images/Normal-8170.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8170.png,Normal/masks/Normal-8170.png,True,False,299,299,L,153.75,a208102c277587c09d4061e0922ac9ea
Normal-8171.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8171.png,Normal/images/Normal-8171.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8171.png,Normal/masks/Normal-8171.png,True,False,299,299,L,91.74,b97c7cdb1e9261a8f1ac2603ec47c4a1
Normal-8172.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8172.png,Normal/images/Normal-8172.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8172.png,Normal/masks/Normal-8172.png,True,False,299,299,L,154.0,8c12a0c1cdec4d9110b81bad9d57cb63
Normal-8173.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8173.png,Normal/images/Normal-8173.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8173.png,Normal/masks/Normal-8173.png,True,False,299,299,L,138.88,2f54648ec4b1714552fef2244df48ed0
Normal-8174.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8174.png,Normal/images/Normal-8174.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8174.png,Normal/masks/Normal-8174.png,True,False,299,299,L,98.94,862b6c3937cfebb66ed896b6cb0a2280
Normal-8175.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8175.png,Normal/images/Normal-8175.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8175.png,Normal/masks/Normal-8175.png,True,False,299,299,L,110.3,c2761f25825db4f8bf341b9bcc812f38
Normal-8176.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8176.png,Normal/images/Normal-8176.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8176.png,Normal/masks/Normal-8176.png,True,False,299,299,L,118.05,1f7f13de60e90be2116e74147cd21fae
Normal-8177.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8177.png,Normal/images/Normal-8177.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8177.png,Normal/masks/Normal-8177.png,True,False,299,299,L,96.73,5ac67a3bf7b68bb6097371091aa918bf
Normal-8178.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8178.png,Normal/images/Normal-8178.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8178.png,Normal/masks/Normal-8178.png,True,False,299,299,L,99.49,35e1d60b74f900e87386f937cc404bd0
Normal-8179.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8179.png,Normal/images/Normal-8179.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8179.png,Normal/masks/Normal-8179.png,True,False,299,299,L,122.64,166904ae4769b7631a7048ce24fa81f3
Normal-818.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-818.png,Normal/images/Normal-818.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-818.png,Normal/masks/Normal-818.png,True,False,299,299,L,164.0,afe5ddfc8aa54b115eedd6c5d18ea439
Normal-8180.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8180.png,Normal/images/Normal-8180.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8180.png,Normal/masks/Normal-8180.png,True,False,299,299,L,110.2,d794a7ca82267c9a0f27b2e7a7a6c34d
Normal-8181.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8181.png,Normal/images/Normal-8181.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8181.png,Normal/masks/Normal-8181.png,True,False,299,299,L,92.73,c03ed6c5c45d90cc0e6e4669a187bb35
Normal-8182.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8182.png,Normal/images/Normal-8182.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8182.png,Normal/masks/Normal-8182.png,True,False,299,299,L,97.29,d68598c95a62e37ab3b41c494097116a
Normal-8183.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8183.png,Normal/images/Normal-8183.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8183.png,Normal/masks/Normal-8183.png,True,False,299,299,L,160.77,71e208c33d553c08c7e6df092fa8cbd1
Normal-8184.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8184.png,Normal/images/Normal-8184.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8184.png,Normal/masks/Normal-8184.png,True,False,299,299,L,111.43,9eb98d81b1018ecb354c4cb202ddafa4
Normal-8185.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8185.png,Normal/images/Normal-8185.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8185.png,Normal/masks/Normal-8185.png,True,False,299,299,L,136.96,076c2fc7288efbac294414bd36496b3d
Normal-8186.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8186.png,Normal/images/Normal-8186.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8186.png,Normal/masks/Normal-8186.png,True,False,299,299,L,163.69,9a3a91186c4d8f8011daabfd134a978a
Normal-8187.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8187.png,Normal/images/Normal-8187.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8187.png,Normal/masks/Normal-8187.png,True,False,299,299,L,103.91,11b577846e3c0b17bb86620297435ad2
Normal-8188.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8188.png,Normal/images/Normal-8188.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8188.png,Normal/masks/Normal-8188.png,True,False,299,299,L,124.4,413d92667f7e49b8de59af92f4e58279
Normal-8189.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8189.png,Normal/images/Normal-8189.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8189.png,Normal/masks/Normal-8189.png,True,False,299,299,L,91.02,e20c029fe13182b7718f819a126fc7bd
Normal-819.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-819.png,Normal/images/Normal-819.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-819.png,Normal/masks/Normal-819.png,True,False,299,299,L,103.57,15d9d48689e1bcafb34459590e8beff9
Normal-8190.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8190.png,Normal/images/Normal-8190.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8190.png,Normal/masks/Normal-8190.png,True,False,299,299,L,123.19,1e12da3ca5a220ea794e32286aca7ea6
Normal-8191.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8191.png,Normal/images/Normal-8191.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8191.png,Normal/masks/Normal-8191.png,True,False,299,299,L,142.95,622d69470197ab811a3b18f550ff4a91
Normal-8192.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8192.png,Normal/images/Normal-8192.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8192.png,Normal/masks/Normal-8192.png,True,False,299,299,L,139.14,5d46bcb2714c97616cc49a1d190bdffb
Normal-8193.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8193.png,Normal/images/Normal-8193.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8193.png,Normal/masks/Normal-8193.png,True,False,299,299,L,132.73,c6b5515753109ff98b4deae0207592c8
Normal-8194.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8194.png,Normal/images/Normal-8194.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8194.png,Normal/masks/Normal-8194.png,True,False,299,299,L,127.71,76826f22ed86b982a96cc3a748722cfc
Normal-8195.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8195.png,Normal/images/Normal-8195.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8195.png,Normal/masks/Normal-8195.png,True,False,299,299,L,166.18,745bd077377ac64c9f4d371a512603df
Normal-8196.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8196.png,Normal/images/Normal-8196.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8196.png,Normal/masks/Normal-8196.png,True,False,299,299,L,126.3,e4d4749daa7a316bf142de071220200e
Normal-8197.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8197.png,Normal/images/Normal-8197.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8197.png,Normal/masks/Normal-8197.png,True,False,299,299,L,133.67,7d9174dc164b423ab91a1fe993d4124e
Normal-8198.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8198.png,Normal/images/Normal-8198.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8198.png,Normal/masks/Normal-8198.png,True,False,299,299,L,120.5,47834b2a8c764864fc547d19e218074f
Normal-8199.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8199.png,Normal/images/Normal-8199.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8199.png,Normal/masks/Normal-8199.png,True,False,299,299,L,109.54,c1290134a8de350d3cf34481d8999283
Normal-82.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-82.png,Normal/images/Normal-82.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-82.png,Normal/masks/Normal-82.png,True,False,299,299,L,129.01,6ce6e376fe5bd9ad9a4c3ee1ce21a116
Normal-820.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-820.png,Normal/images/Normal-820.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-820.png,Normal/masks/Normal-820.png,True,False,299,299,L,99.8,5bde1184d09ad1238e63f6c44010d4f6
Normal-8200.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8200.png,Normal/images/Normal-8200.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8200.png,Normal/masks/Normal-8200.png,True,False,299,299,L,129.44,a0ec01826d4974117a7633f9cb642413
Normal-8201.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8201.png,Normal/images/Normal-8201.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8201.png,Normal/masks/Normal-8201.png,True,False,299,299,L,127.29,dd18f0b49f9c3ee70c398e1831d1ade1
Normal-8202.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8202.png,Normal/images/Normal-8202.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8202.png,Normal/masks/Normal-8202.png,True,False,299,299,L,112.87,a480320446ec38ff19257f3de746ec85
Normal-8203.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8203.png,Normal/images/Normal-8203.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8203.png,Normal/masks/Normal-8203.png,True,False,299,299,L,160.0,a6f0c6ed0e0eb93ff5c30119bf461d14
Normal-8204.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8204.png,Normal/images/Normal-8204.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8204.png,Normal/masks/Normal-8204.png,True,False,299,299,L,95.12,1d21260e2a9739f35ea414ff8e7f62c1
Normal-8205.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8205.png,Normal/images/Normal-8205.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8205.png,Normal/masks/Normal-8205.png,True,False,299,299,L,165.38,ed5129b89e181e5095349bb1d043baf6
Normal-8206.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8206.png,Normal/images/Normal-8206.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8206.png,Normal/masks/Normal-8206.png,True,False,299,299,L,148.38,c954274adb187a9d80f18075e0b7e50c
Normal-8207.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8207.png,Normal/images/Normal-8207.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8207.png,Normal/masks/Normal-8207.png,True,False,299,299,L,113.7,6154a3660fbd579f4b37845fe02a51cb
Normal-8208.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8208.png,Normal/images/Normal-8208.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8208.png,Normal/masks/Normal-8208.png,True,False,299,299,L,180.26,c7ef21c43b3496bb34704228aa09ba9e
Normal-8209.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8209.png,Normal/images/Normal-8209.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8209.png,Normal/masks/Normal-8209.png,True,False,299,299,L,105.21,9061da554cfae517b1555bce85d5b008
Normal-821.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-821.png,Normal/images/Normal-821.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-821.png,Normal/masks/Normal-821.png,True,False,299,299,L,137.84,c6b87bf243fbbf910ade7d49a5f4de21
Normal-8210.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8210.png,Normal/images/Normal-8210.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8210.png,Normal/masks/Normal-8210.png,True,False,299,299,L,122.33,f554546b610a8ac1e2aabba206a82dbf
Normal-8211.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8211.png,Normal/images/Normal-8211.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8211.png,Normal/masks/Normal-8211.png,True,False,299,299,L,149.76,cecdb13c35b1195ea903b659daa87c45
Normal-8212.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8212.png,Normal/images/Normal-8212.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8212.png,Normal/masks/Normal-8212.png,True,False,299,299,L,176.69,f9461c2b91f512c5c4f86d8929f382de
Normal-8213.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8213.png,Normal/images/Normal-8213.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8213.png,Normal/masks/Normal-8213.png,True,False,299,299,L,124.49,f642a005cb5c354efafc2d48114dc6e4
Normal-8214.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8214.png,Normal/images/Normal-8214.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8214.png,Normal/masks/Normal-8214.png,True,False,299,299,L,103.06,135745987662f2241e2d10e349284eae
Normal-8215.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8215.png,Normal/images/Normal-8215.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8215.png,Normal/masks/Normal-8215.png,True,False,299,299,L,173.09,f62225abdaf51f311463c5aba330bf2e
Normal-8216.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8216.png,Normal/images/Normal-8216.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8216.png,Normal/masks/Normal-8216.png,True,False,299,299,L,134.76,8fe643fd093244b2a37f284b95db1a8a
Normal-8217.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8217.png,Normal/images/Normal-8217.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8217.png,Normal/masks/Normal-8217.png,True,False,299,299,L,144.72,ce4a644b5efb4d7841ffa5d30b14d673
Normal-8218.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8218.png,Normal/images/Normal-8218.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8218.png,Normal/masks/Normal-8218.png,True,False,299,299,L,133.86,780da9439a9ae775843ec557c4ffdfec
Normal-8219.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8219.png,Normal/images/Normal-8219.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8219.png,Normal/masks/Normal-8219.png,True,False,299,299,L,77.32,5100aa2cf234d4bf4d7f2f181fd3440b
Normal-822.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-822.png,Normal/images/Normal-822.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-822.png,Normal/masks/Normal-822.png,True,False,299,299,L,125.27,f04fac66fb4da93782322830ca550707
Normal-8220.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8220.png,Normal/images/Normal-8220.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8220.png,Normal/masks/Normal-8220.png,True,False,299,299,L,134.56,4634db58e06e3fce3ea92b7f38944833
Normal-8221.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8221.png,Normal/images/Normal-8221.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8221.png,Normal/masks/Normal-8221.png,True,False,299,299,L,132.37,eb8bdabe09488f5bc29bf083e5eb4c35
Normal-8222.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8222.png,Normal/images/Normal-8222.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8222.png,Normal/masks/Normal-8222.png,True,False,299,299,L,170.76,215676b5e94d6de851312e06d0728414
Normal-8223.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8223.png,Normal/images/Normal-8223.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8223.png,Normal/masks/Normal-8223.png,True,False,299,299,L,155.8,8f62c27aabefbe1a4cb1706651d62cb3
Normal-8224.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8224.png,Normal/images/Normal-8224.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8224.png,Normal/masks/Normal-8224.png,True,False,299,299,L,138.3,d25216b8752eff78a6e0829a710a2d72
Normal-8225.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8225.png,Normal/images/Normal-8225.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8225.png,Normal/masks/Normal-8225.png,True,False,299,299,L,118.23,c32037e5bb8189f5636ba5fbf8f2699c
Normal-8226.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8226.png,Normal/images/Normal-8226.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8226.png,Normal/masks/Normal-8226.png,True,False,299,299,L,118.01,56625ebde73c911499eff69e3a22237a
Normal-8227.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8227.png,Normal/images/Normal-8227.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8227.png,Normal/masks/Normal-8227.png,True,False,299,299,L,167.64,375c5c70cc2da6d88d345fce15f2964f
Normal-8228.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8228.png,Normal/images/Normal-8228.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8228.png,Normal/masks/Normal-8228.png,True,False,299,299,L,113.17,5862b68f299841fe18ffefbce77aae7c
Normal-8229.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8229.png,Normal/images/Normal-8229.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8229.png,Normal/masks/Normal-8229.png,True,False,299,299,L,129.31,9633cfbd78739e2b201d83cc58f808a2
Normal-823.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-823.png,Normal/images/Normal-823.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-823.png,Normal/masks/Normal-823.png,True,False,299,299,L,134.87,6cb118fae3dbb5154bc9a064ad4c6490
Normal-8230.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8230.png,Normal/images/Normal-8230.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8230.png,Normal/masks/Normal-8230.png,True,False,299,299,L,133.22,8fa0384764598828444021e9e5bb37d2
Normal-8231.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8231.png,Normal/images/Normal-8231.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8231.png,Normal/masks/Normal-8231.png,True,False,299,299,L,131.07,417cab9fcc498c73b98661dc737ee794
Normal-8232.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8232.png,Normal/images/Normal-8232.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8232.png,Normal/masks/Normal-8232.png,True,False,299,299,L,102.49,5ccadf38a7e65f7e6adfd38180fc4a34
Normal-8233.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8233.png,Normal/images/Normal-8233.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8233.png,Normal/masks/Normal-8233.png,True,False,299,299,L,129.94,c36031497d52f7250a4041364fb53f7f
Normal-8234.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8234.png,Normal/images/Normal-8234.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8234.png,Normal/masks/Normal-8234.png,True,False,299,299,L,116.6,d5d74eb68e0803d3eae3a6167015db87
Normal-8235.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8235.png,Normal/images/Normal-8235.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8235.png,Normal/masks/Normal-8235.png,True,False,299,299,L,100.99,a26ecc33cfd2b5897f4402ce49c4dc29
Normal-8236.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8236.png,Normal/images/Normal-8236.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8236.png,Normal/masks/Normal-8236.png,True,False,299,299,L,115.81,aaf68a04c99bd2aa6056c1954c828caa
Normal-8237.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8237.png,Normal/images/Normal-8237.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8237.png,Normal/masks/Normal-8237.png,True,False,299,299,L,105.14,daf0fdf8b72c21f1de6de915cda1fd3a
Normal-8238.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8238.png,Normal/images/Normal-8238.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8238.png,Normal/masks/Normal-8238.png,True,False,299,299,L,103.48,876c402171d8468213fc22ab8dd25ba8
Normal-8239.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8239.png,Normal/images/Normal-8239.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8239.png,Normal/masks/Normal-8239.png,True,False,299,299,L,155.06,51c2c6c317f9b589596af5d60ca84ee6
Normal-824.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-824.png,Normal/images/Normal-824.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-824.png,Normal/masks/Normal-824.png,True,False,299,299,L,133.64,e233b1112b6684e202f3b614ba3cf8e0
Normal-8240.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8240.png,Normal/images/Normal-8240.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8240.png,Normal/masks/Normal-8240.png,True,False,299,299,L,166.49,3cb6300b396259107ad89482ba525ce4
Normal-8241.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8241.png,Normal/images/Normal-8241.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8241.png,Normal/masks/Normal-8241.png,True,False,299,299,L,180.29,eb747f9be3fd027a85637c5db84cd692
Normal-8242.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8242.png,Normal/images/Normal-8242.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8242.png,Normal/masks/Normal-8242.png,True,False,299,299,L,160.31,5d80237e60f60ba8cce586b9b0ae5871
Normal-8243.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8243.png,Normal/images/Normal-8243.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8243.png,Normal/masks/Normal-8243.png,True,False,299,299,L,106.96,029745641d62fa65042228f28b15cd08
Normal-8244.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8244.png,Normal/images/Normal-8244.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8244.png,Normal/masks/Normal-8244.png,True,False,299,299,L,172.39,35cd29672725e667f00ea2d644ca0958
Normal-8245.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8245.png,Normal/images/Normal-8245.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8245.png,Normal/masks/Normal-8245.png,True,False,299,299,L,143.79,7ad58cfd7d81fad4dc7ade9534ee47e6
Normal-8246.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8246.png,Normal/images/Normal-8246.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8246.png,Normal/masks/Normal-8246.png,True,False,299,299,L,107.93,84537135d8afccbcf3063271c0bfd54f
Normal-8247.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8247.png,Normal/images/Normal-8247.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8247.png,Normal/masks/Normal-8247.png,True,False,299,299,L,110.54,49718caadba55734b90447e684ea39af
Normal-8248.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8248.png,Normal/images/Normal-8248.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8248.png,Normal/masks/Normal-8248.png,True,False,299,299,L,177.34,a510c4bee943bd12927658b774eb521d
Normal-8249.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8249.png,Normal/images/Normal-8249.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8249.png,Normal/masks/Normal-8249.png,True,False,299,299,L,121.16,181c0a3ba2ca01b32cc4f26383cd300e
Normal-825.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-825.png,Normal/images/Normal-825.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-825.png,Normal/masks/Normal-825.png,True,False,299,299,L,125.27,cf2a559561ad2bd3f1b5a3956e541e6d
Normal-8250.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8250.png,Normal/images/Normal-8250.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8250.png,Normal/masks/Normal-8250.png,True,False,299,299,L,167.89,4fb3f92504b7d73f52520e1d658efc9f
Normal-8251.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8251.png,Normal/images/Normal-8251.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8251.png,Normal/masks/Normal-8251.png,True,False,299,299,L,124.07,9f06ff7a8c40c5848f1fb4bc871d7d6d
Normal-8252.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8252.png,Normal/images/Normal-8252.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8252.png,Normal/masks/Normal-8252.png,True,False,299,299,L,124.01,71e97df89f7a45404ff26cc074aad1d2
Normal-8253.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8253.png,Normal/images/Normal-8253.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8253.png,Normal/masks/Normal-8253.png,True,False,299,299,L,121.45,2bd4964b7b7e2422b7c7849b8ddc4aee
Normal-8254.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8254.png,Normal/images/Normal-8254.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8254.png,Normal/masks/Normal-8254.png,True,False,299,299,L,124.53,83c3d61239ea7f8b5f306ddb0b0e0e40
Normal-8255.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8255.png,Normal/images/Normal-8255.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8255.png,Normal/masks/Normal-8255.png,True,False,299,299,L,113.57,4fd9118cc57e9944332b1c1f465f9259
Normal-8256.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8256.png,Normal/images/Normal-8256.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8256.png,Normal/masks/Normal-8256.png,True,False,299,299,L,102.91,d757137953c0ac19ce00535d39687346
Normal-8257.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8257.png,Normal/images/Normal-8257.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8257.png,Normal/masks/Normal-8257.png,True,False,299,299,L,147.03,c1a098571863c61a73c255078d99c4e0
Normal-8258.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8258.png,Normal/images/Normal-8258.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8258.png,Normal/masks/Normal-8258.png,True,False,299,299,L,118.07,ddf8999ef9bcee4ea883f43190c4ca52
Normal-8259.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8259.png,Normal/images/Normal-8259.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8259.png,Normal/masks/Normal-8259.png,True,False,299,299,L,132.52,be8312474277b1c8d74c2273052e53ba
Normal-826.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-826.png,Normal/images/Normal-826.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-826.png,Normal/masks/Normal-826.png,True,False,299,299,L,123.74,c177b1085f5628bde1a5edbd425b8f5a
Normal-8260.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8260.png,Normal/images/Normal-8260.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8260.png,Normal/masks/Normal-8260.png,True,False,299,299,L,161.69,4a91ac4ffa12af67636296cf02e0e874
Normal-8261.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8261.png,Normal/images/Normal-8261.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8261.png,Normal/masks/Normal-8261.png,True,False,299,299,L,146.77,29ea4f5d6c5666c83c1fb11dce54d4dd
Normal-8262.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8262.png,Normal/images/Normal-8262.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8262.png,Normal/masks/Normal-8262.png,True,False,299,299,L,99.05,eced9ac66623e22e70a26c1dfdf574b6
Normal-8263.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8263.png,Normal/images/Normal-8263.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8263.png,Normal/masks/Normal-8263.png,True,False,299,299,L,124.56,20937effdba99077527ffef9d9bc2cfd
Normal-8264.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8264.png,Normal/images/Normal-8264.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8264.png,Normal/masks/Normal-8264.png,True,False,299,299,L,176.22,c0dce22094e8132ee0f242cfe6d61e39
Normal-8265.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8265.png,Normal/images/Normal-8265.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8265.png,Normal/masks/Normal-8265.png,True,False,299,299,L,107.76,70a859baae9d3a999e43f1bac384ade7
Normal-8266.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8266.png,Normal/images/Normal-8266.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8266.png,Normal/masks/Normal-8266.png,True,False,299,299,L,105.61,2cbdb52c1679394f3c4a653b8a934ff6
Normal-8267.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8267.png,Normal/images/Normal-8267.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8267.png,Normal/masks/Normal-8267.png,True,False,299,299,L,95.1,083b6b16f43dbbb11ce03a1280186691
Normal-8268.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8268.png,Normal/images/Normal-8268.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8268.png,Normal/masks/Normal-8268.png,True,False,299,299,L,129.81,af8351a56b9fcd5b6e9a5905b1f39dc9
Normal-8269.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8269.png,Normal/images/Normal-8269.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8269.png,Normal/masks/Normal-8269.png,True,False,299,299,L,164.55,012c639adb97bf7abf93345874ed62cf
Normal-827.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-827.png,Normal/images/Normal-827.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-827.png,Normal/masks/Normal-827.png,True,False,299,299,L,116.51,a87f61e150c6e231f4c4e072db10f722
Normal-8270.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8270.png,Normal/images/Normal-8270.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8270.png,Normal/masks/Normal-8270.png,True,False,299,299,L,134.78,128d78f4d0cfc758eb14f102c28ec305
Normal-8271.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8271.png,Normal/images/Normal-8271.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8271.png,Normal/masks/Normal-8271.png,True,False,299,299,L,132.99,c44fc3eb71bedf17773f7c4c414154f2
Normal-8272.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8272.png,Normal/images/Normal-8272.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8272.png,Normal/masks/Normal-8272.png,True,False,299,299,L,115.24,4df949fc5da115bb86a918491b52c06c
Normal-8273.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8273.png,Normal/images/Normal-8273.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8273.png,Normal/masks/Normal-8273.png,True,False,299,299,L,104.9,c91ce987d8bf083c849fe108f4aab5e8
Normal-8274.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8274.png,Normal/images/Normal-8274.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8274.png,Normal/masks/Normal-8274.png,True,False,299,299,L,103.14,717d7fb9ffcb4b6146d3004e0f660dd9
Normal-8275.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8275.png,Normal/images/Normal-8275.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8275.png,Normal/masks/Normal-8275.png,True,False,299,299,L,189.63,3e4de8d90bd86965d04f42df865b0300
Normal-8276.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8276.png,Normal/images/Normal-8276.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8276.png,Normal/masks/Normal-8276.png,True,False,299,299,L,128.06,50d1a691c9b627fe003e75426c85b5ab
Normal-8277.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8277.png,Normal/images/Normal-8277.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8277.png,Normal/masks/Normal-8277.png,True,False,299,299,L,131.81,23390ef4999889d1d0656b9efed4fe1f
Normal-8278.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8278.png,Normal/images/Normal-8278.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8278.png,Normal/masks/Normal-8278.png,True,False,299,299,L,142.26,73bf627bc9c67fa53b2d3ac12e56bd25
Normal-8279.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8279.png,Normal/images/Normal-8279.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8279.png,Normal/masks/Normal-8279.png,True,False,299,299,L,123.65,1a987b06bcb5adc743eab027d1a2ab20
Normal-828.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-828.png,Normal/images/Normal-828.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-828.png,Normal/masks/Normal-828.png,True,False,299,299,L,137.6,205c06ed845f58641fdfe105547d4122
Normal-8280.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8280.png,Normal/images/Normal-8280.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8280.png,Normal/masks/Normal-8280.png,True,False,299,299,L,150.42,b522b5da39d7ac6b226fcb944a54820d
Normal-8281.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8281.png,Normal/images/Normal-8281.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8281.png,Normal/masks/Normal-8281.png,True,False,299,299,L,122.66,2229d7a3049b0bc62b19a01a144ce40f
Normal-8282.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8282.png,Normal/images/Normal-8282.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8282.png,Normal/masks/Normal-8282.png,True,False,299,299,L,109.21,5e4346fcf417efae24f7832f9ae3715a
Normal-8283.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8283.png,Normal/images/Normal-8283.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8283.png,Normal/masks/Normal-8283.png,True,False,299,299,L,130.97,8ce021843ac423dedaa8b19a9d4d4f1c
Normal-8284.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8284.png,Normal/images/Normal-8284.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8284.png,Normal/masks/Normal-8284.png,True,False,299,299,L,109.3,6c832d7851fc97c34e13211db6e5f60b
Normal-8285.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8285.png,Normal/images/Normal-8285.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8285.png,Normal/masks/Normal-8285.png,True,False,299,299,L,111.36,576275bcf4cb6a7e9bbed1e0f624e64b
Normal-8286.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8286.png,Normal/images/Normal-8286.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8286.png,Normal/masks/Normal-8286.png,True,False,299,299,L,106.16,8fcc7b819ab43e7148c3a9627b17b68c
Normal-8287.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8287.png,Normal/images/Normal-8287.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8287.png,Normal/masks/Normal-8287.png,True,False,299,299,L,120.09,9531567af2f1bc00eeaee4c089e4a400
Normal-8288.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8288.png,Normal/images/Normal-8288.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8288.png,Normal/masks/Normal-8288.png,True,False,299,299,L,162.89,6a3d52df943974bba7aad1fa309b7cc9
Normal-8289.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8289.png,Normal/images/Normal-8289.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8289.png,Normal/masks/Normal-8289.png,True,False,299,299,L,129.4,7dd8c76f035d739a67676263f9aca7e1
Normal-829.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-829.png,Normal/images/Normal-829.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-829.png,Normal/masks/Normal-829.png,True,False,299,299,L,116.85,e46b6f9cd08f28596b7a671638c3a327
Normal-8290.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8290.png,Normal/images/Normal-8290.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8290.png,Normal/masks/Normal-8290.png,True,False,299,299,L,116.87,53eacdce96f8114094e63aa79f9a7693
Normal-8291.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8291.png,Normal/images/Normal-8291.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8291.png,Normal/masks/Normal-8291.png,True,False,299,299,L,136.33,eafc89091fb7694b80653d1bdcb6e03b
Normal-8292.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8292.png,Normal/images/Normal-8292.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8292.png,Normal/masks/Normal-8292.png,True,False,299,299,L,163.17,8bf852eebcd3cce8dbfeebc095e39438
Normal-8293.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8293.png,Normal/images/Normal-8293.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8293.png,Normal/masks/Normal-8293.png,True,False,299,299,L,131.09,f2b43f0623a1f9b5a5fa21571a51ce1b
Normal-8294.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8294.png,Normal/images/Normal-8294.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8294.png,Normal/masks/Normal-8294.png,True,False,299,299,L,135.67,b76174a4d7d8c44314c1eb13714f3255
Normal-8295.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8295.png,Normal/images/Normal-8295.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8295.png,Normal/masks/Normal-8295.png,True,False,299,299,L,150.88,ba0b0e94660d5b8ae8b37703268aee74
Normal-8296.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8296.png,Normal/images/Normal-8296.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8296.png,Normal/masks/Normal-8296.png,True,False,299,299,L,145.69,28ea16c3d41740aa12d161d131625107
Normal-8297.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8297.png,Normal/images/Normal-8297.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8297.png,Normal/masks/Normal-8297.png,True,False,299,299,L,126.15,0f75bdaacd10c93e42d108a01128dd36
Normal-8298.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8298.png,Normal/images/Normal-8298.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8298.png,Normal/masks/Normal-8298.png,True,False,299,299,L,134.45,da625d20ccc6489de039b1f7ef236039
Normal-8299.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8299.png,Normal/images/Normal-8299.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8299.png,Normal/masks/Normal-8299.png,True,False,299,299,L,111.0,e80f3b1c0d7afa475bf010b10401e521
Normal-83.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-83.png,Normal/images/Normal-83.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-83.png,Normal/masks/Normal-83.png,True,False,299,299,L,110.08,ebd03a452418e0d1629ef8884fc74b27
Normal-830.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-830.png,Normal/images/Normal-830.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-830.png,Normal/masks/Normal-830.png,True,False,299,299,L,114.48,af0726bb31949540857c5c9334401ce8
Normal-8300.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8300.png,Normal/images/Normal-8300.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8300.png,Normal/masks/Normal-8300.png,True,False,299,299,L,104.18,fec38717230f2292d74e4074a9c0bb02
Normal-8301.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8301.png,Normal/images/Normal-8301.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8301.png,Normal/masks/Normal-8301.png,True,False,299,299,L,112.22,7cfbdca6f6f5ad350158f8cb9fa25817
Normal-8302.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8302.png,Normal/images/Normal-8302.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8302.png,Normal/masks/Normal-8302.png,True,False,299,299,L,137.5,071d7a9ba79d532403f41c32bff11ef5
Normal-8303.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8303.png,Normal/images/Normal-8303.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8303.png,Normal/masks/Normal-8303.png,True,False,299,299,L,163.29,3025a07bc579428fd483ea7384380a34
Normal-8304.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8304.png,Normal/images/Normal-8304.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8304.png,Normal/masks/Normal-8304.png,True,False,299,299,L,174.91,58141c8c3e407f8df884ce5ebe949bf9
Normal-8305.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8305.png,Normal/images/Normal-8305.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8305.png,Normal/masks/Normal-8305.png,True,False,299,299,L,135.1,60fcd53ba44a8b265beb495a0dd97da9
Normal-8306.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8306.png,Normal/images/Normal-8306.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8306.png,Normal/masks/Normal-8306.png,True,False,299,299,L,111.74,d1b918227c5b07840db0514f0cdb96e1
Normal-8307.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8307.png,Normal/images/Normal-8307.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8307.png,Normal/masks/Normal-8307.png,True,False,299,299,L,101.1,4e9ee74819139597f11ea432abf1fc22
Normal-8308.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8308.png,Normal/images/Normal-8308.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8308.png,Normal/masks/Normal-8308.png,True,False,299,299,L,94.76,c2c46489eb486b9603389f27efe39d70
Normal-8309.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8309.png,Normal/images/Normal-8309.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8309.png,Normal/masks/Normal-8309.png,True,False,299,299,L,111.75,4372a97123aa6c885a223015a8e80c3b
Normal-831.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-831.png,Normal/images/Normal-831.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-831.png,Normal/masks/Normal-831.png,True,False,299,299,L,120.46,f28242e84061d0be22c99a0118611651
Normal-8310.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8310.png,Normal/images/Normal-8310.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8310.png,Normal/masks/Normal-8310.png,True,False,299,299,L,100.92,e1c1359ae26c8fab42c3cddcd26de1da
Normal-8311.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8311.png,Normal/images/Normal-8311.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8311.png,Normal/masks/Normal-8311.png,True,False,299,299,L,151.19,cadcadcf9e05c1b876120d7cc6dd2c88
Normal-8312.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8312.png,Normal/images/Normal-8312.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8312.png,Normal/masks/Normal-8312.png,True,False,299,299,L,120.03,0a45417335e671f19d2072398f8c35d7
Normal-8313.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8313.png,Normal/images/Normal-8313.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8313.png,Normal/masks/Normal-8313.png,True,False,299,299,L,169.15,de714826f9fe6b6a6da9e577e6522ce4
Normal-8314.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8314.png,Normal/images/Normal-8314.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8314.png,Normal/masks/Normal-8314.png,True,False,299,299,L,145.86,bc57e07be12b5e9000b664a124e71825
Normal-8315.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8315.png,Normal/images/Normal-8315.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8315.png,Normal/masks/Normal-8315.png,True,False,299,299,L,173.17,87aa6d2395c821702780062f82c08c66
Normal-8316.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8316.png,Normal/images/Normal-8316.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8316.png,Normal/masks/Normal-8316.png,True,False,299,299,L,48.63,614d475a8bfef3b632abf7e539d6e387
Normal-8317.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8317.png,Normal/images/Normal-8317.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8317.png,Normal/masks/Normal-8317.png,True,False,299,299,L,136.92,76e1c8c175f0aec999b238212c1f7c01
Normal-8318.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8318.png,Normal/images/Normal-8318.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8318.png,Normal/masks/Normal-8318.png,True,False,299,299,L,90.05,318d3db03e16e54a02db47195342eba9
Normal-8319.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8319.png,Normal/images/Normal-8319.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8319.png,Normal/masks/Normal-8319.png,True,False,299,299,L,102.1,b0ae979a94f1eccc46020a2a75143ce4
Normal-832.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-832.png,Normal/images/Normal-832.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-832.png,Normal/masks/Normal-832.png,True,False,299,299,L,132.96,b2f5532c41ee33914749c6023213512c
Normal-8320.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8320.png,Normal/images/Normal-8320.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8320.png,Normal/masks/Normal-8320.png,True,False,299,299,L,130.38,3519d3048a956457d7a2396fe5ceec0f
Normal-8321.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8321.png,Normal/images/Normal-8321.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8321.png,Normal/masks/Normal-8321.png,True,False,299,299,L,119.56,3732cdc7de34c5515ea4fb1b15a05d26
Normal-8322.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8322.png,Normal/images/Normal-8322.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8322.png,Normal/masks/Normal-8322.png,True,False,299,299,L,107.88,11a65a06598108b80fd0ced460d067af
Normal-8323.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8323.png,Normal/images/Normal-8323.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8323.png,Normal/masks/Normal-8323.png,True,False,299,299,L,116.56,e0f8d1e70fa2882a301a690df06b0ee5
Normal-8324.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8324.png,Normal/images/Normal-8324.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8324.png,Normal/masks/Normal-8324.png,True,False,299,299,L,110.19,0ea81fa79d0f335a54ce70020a496151
Normal-8325.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8325.png,Normal/images/Normal-8325.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8325.png,Normal/masks/Normal-8325.png,True,False,299,299,L,114.81,2282671332a011112c03aa17be625dd6
Normal-8326.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8326.png,Normal/images/Normal-8326.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8326.png,Normal/masks/Normal-8326.png,True,False,299,299,L,172.89,f690d85d5f883e8e070b2b109fdb412a
Normal-8327.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8327.png,Normal/images/Normal-8327.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8327.png,Normal/masks/Normal-8327.png,True,False,299,299,L,171.2,c95d1c1eb8fd10d508a5c4181198c7a5
Normal-8328.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8328.png,Normal/images/Normal-8328.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8328.png,Normal/masks/Normal-8328.png,True,False,299,299,L,169.69,d57b023fb09b0144edb597e43365b95e
Normal-8329.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8329.png,Normal/images/Normal-8329.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8329.png,Normal/masks/Normal-8329.png,True,False,299,299,L,78.48,36f63832b4d9b1cbd36626c5c0db750d
Normal-833.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-833.png,Normal/images/Normal-833.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-833.png,Normal/masks/Normal-833.png,True,False,299,299,L,145.02,635b7275d515b88188228ca5684d569b
Normal-8330.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8330.png,Normal/images/Normal-8330.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8330.png,Normal/masks/Normal-8330.png,True,False,299,299,L,124.13,641ba94b7636748d66e14dde6ecc1b28
Normal-8331.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8331.png,Normal/images/Normal-8331.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8331.png,Normal/masks/Normal-8331.png,True,False,299,299,L,123.43,41c60be63da6483ea3290a8b3c96a337
Normal-8332.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8332.png,Normal/images/Normal-8332.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8332.png,Normal/masks/Normal-8332.png,True,False,299,299,L,175.04,4fea18487f43c6f46552f74306bf3c21
Normal-8333.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8333.png,Normal/images/Normal-8333.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8333.png,Normal/masks/Normal-8333.png,True,False,299,299,L,133.93,2616066266bd547beecfafa2f599a47a
Normal-8334.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8334.png,Normal/images/Normal-8334.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8334.png,Normal/masks/Normal-8334.png,True,False,299,299,L,160.12,ecfb9af2301c03c8fc0247a2d4c113c3
Normal-8335.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8335.png,Normal/images/Normal-8335.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8335.png,Normal/masks/Normal-8335.png,True,False,299,299,L,110.92,a15c20d73cfdbccf0c3f18108e40da87
Normal-8336.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8336.png,Normal/images/Normal-8336.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8336.png,Normal/masks/Normal-8336.png,True,False,299,299,L,110.81,c62b2d0c07efe64383341f4df53692c4
Normal-8337.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8337.png,Normal/images/Normal-8337.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8337.png,Normal/masks/Normal-8337.png,True,False,299,299,L,118.37,978dd8d4b527c1b3e96ba7f44e47ae2c
Normal-8338.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8338.png,Normal/images/Normal-8338.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8338.png,Normal/masks/Normal-8338.png,True,False,299,299,L,120.14,c807f78068f6b27d9226eaa40af858bf
Normal-8339.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8339.png,Normal/images/Normal-8339.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8339.png,Normal/masks/Normal-8339.png,True,False,299,299,L,122.24,4bf85254986b595fd950d94322670dd5
Normal-834.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-834.png,Normal/images/Normal-834.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-834.png,Normal/masks/Normal-834.png,True,False,299,299,L,126.19,76f52c332af10a944046aa21901191e2
Normal-8340.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8340.png,Normal/images/Normal-8340.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8340.png,Normal/masks/Normal-8340.png,True,False,299,299,L,158.05,b7f8c74d1b482f8871e587ed1bf6f625
Normal-8341.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8341.png,Normal/images/Normal-8341.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8341.png,Normal/masks/Normal-8341.png,True,False,299,299,L,99.67,3d4bd5dd97e9002f961111e11566eb72
Normal-8342.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8342.png,Normal/images/Normal-8342.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8342.png,Normal/masks/Normal-8342.png,True,False,299,299,L,157.76,c6843ccc1321ab470901631fbcd2831d
Normal-8343.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8343.png,Normal/images/Normal-8343.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8343.png,Normal/masks/Normal-8343.png,True,False,299,299,L,105.56,276fba126fc13092ca154e5e218687a0
Normal-8344.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8344.png,Normal/images/Normal-8344.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8344.png,Normal/masks/Normal-8344.png,True,False,299,299,L,110.79,7abf75038da0a366cdee318ad2cff699
Normal-8345.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8345.png,Normal/images/Normal-8345.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8345.png,Normal/masks/Normal-8345.png,True,False,299,299,L,112.77,fffc1fdcbbd1bef4326c70f7cef004b8
Normal-8346.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8346.png,Normal/images/Normal-8346.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8346.png,Normal/masks/Normal-8346.png,True,False,299,299,L,109.12,fd6daab405090cc029ba85d21674dc99
Normal-8347.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8347.png,Normal/images/Normal-8347.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8347.png,Normal/masks/Normal-8347.png,True,False,299,299,L,181.99,cc623b5d39282a1e66b8caf257e9e2bf
Normal-8348.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8348.png,Normal/images/Normal-8348.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8348.png,Normal/masks/Normal-8348.png,True,False,299,299,L,128.21,4e64c240c6b0deca0cafe60f47de1169
Normal-8349.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8349.png,Normal/images/Normal-8349.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8349.png,Normal/masks/Normal-8349.png,True,False,299,299,L,119.69,8750fd027e10e589e374ba15bb424f22
Normal-835.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-835.png,Normal/images/Normal-835.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-835.png,Normal/masks/Normal-835.png,True,False,299,299,L,119.98,b0afa796d75a54e0f25477a857d6d7a1
Normal-8350.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8350.png,Normal/images/Normal-8350.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8350.png,Normal/masks/Normal-8350.png,True,False,299,299,L,116.2,0298c6eb1820526b389b5bacd515a522
Normal-8351.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8351.png,Normal/images/Normal-8351.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8351.png,Normal/masks/Normal-8351.png,True,False,299,299,L,150.71,034b053d4e0a5687954c00c53458ea73
Normal-8352.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8352.png,Normal/images/Normal-8352.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8352.png,Normal/masks/Normal-8352.png,True,False,299,299,L,109.43,3f8766b0bd4d615d6eadf94bfd366850
Normal-8353.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8353.png,Normal/images/Normal-8353.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8353.png,Normal/masks/Normal-8353.png,True,False,299,299,L,162.0,e8ba3d563a4928448aded11d957059a1
Normal-8354.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8354.png,Normal/images/Normal-8354.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8354.png,Normal/masks/Normal-8354.png,True,False,299,299,L,85.15,e7cce41c7cd270ed6c7cff4e10eaa9a8
Normal-8355.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8355.png,Normal/images/Normal-8355.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8355.png,Normal/masks/Normal-8355.png,True,False,299,299,L,125.28,e7b4e5370ca43dae04f62ddccc61f2de
Normal-8356.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8356.png,Normal/images/Normal-8356.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8356.png,Normal/masks/Normal-8356.png,True,False,299,299,L,114.58,b02e8fac872fe059adc9c56a1ee61788
Normal-8357.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8357.png,Normal/images/Normal-8357.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8357.png,Normal/masks/Normal-8357.png,True,False,299,299,L,156.14,903d00d5631b0b8f1fd2d40591258d6d
Normal-8358.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8358.png,Normal/images/Normal-8358.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8358.png,Normal/masks/Normal-8358.png,True,False,299,299,L,156.53,6940d851828eb6989be2163f50d9a15e
Normal-8359.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8359.png,Normal/images/Normal-8359.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8359.png,Normal/masks/Normal-8359.png,True,False,299,299,L,190.19,3ef580ca2ea2bde9142be22fb7c0b321
Normal-836.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-836.png,Normal/images/Normal-836.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-836.png,Normal/masks/Normal-836.png,True,False,299,299,L,101.06,11ede42120273895f8392e643d6d2577
Normal-8360.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8360.png,Normal/images/Normal-8360.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8360.png,Normal/masks/Normal-8360.png,True,False,299,299,L,143.55,df2dccbd0ab53b3c3f501ea20a9c5d92
Normal-8361.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8361.png,Normal/images/Normal-8361.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8361.png,Normal/masks/Normal-8361.png,True,False,299,299,L,99.39,a589d1372b9ce4787828ec942cbb0561
Normal-8362.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8362.png,Normal/images/Normal-8362.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8362.png,Normal/masks/Normal-8362.png,True,False,299,299,L,110.23,aeae1c19b3185b6bf4706cbc74f9188b
Normal-8363.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8363.png,Normal/images/Normal-8363.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8363.png,Normal/masks/Normal-8363.png,True,False,299,299,L,114.38,9e40f78f7498c5419c9a3e553546feb5
Normal-8364.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8364.png,Normal/images/Normal-8364.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8364.png,Normal/masks/Normal-8364.png,True,False,299,299,L,172.37,b0b4ddaf95fbf557382d3ae7e13d570a
Normal-8365.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8365.png,Normal/images/Normal-8365.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8365.png,Normal/masks/Normal-8365.png,True,False,299,299,L,105.86,1c5af103d340ee1c2d2e5393cf41de0c
Normal-8366.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8366.png,Normal/images/Normal-8366.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8366.png,Normal/masks/Normal-8366.png,True,False,299,299,L,135.93,3eb1c509943f79b53ea36303d6d8ac54
Normal-8367.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8367.png,Normal/images/Normal-8367.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8367.png,Normal/masks/Normal-8367.png,True,False,299,299,L,117.95,6c7caeadfbd8f607a54451551072031b
Normal-8368.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8368.png,Normal/images/Normal-8368.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8368.png,Normal/masks/Normal-8368.png,True,False,299,299,L,110.79,1a82a43bca35bb76a5f798230cc40062
Normal-8369.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8369.png,Normal/images/Normal-8369.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8369.png,Normal/masks/Normal-8369.png,True,False,299,299,L,161.71,c4fd55fdc419867a7190e56febbfe131
Normal-837.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-837.png,Normal/images/Normal-837.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-837.png,Normal/masks/Normal-837.png,True,False,299,299,L,130.43,3530cf7a5ecb7df986b23a9ae9d90e62
Normal-8370.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8370.png,Normal/images/Normal-8370.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8370.png,Normal/masks/Normal-8370.png,True,False,299,299,L,119.4,03032eb4e750648d52d6c0a752df8ae9
Normal-8371.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8371.png,Normal/images/Normal-8371.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8371.png,Normal/masks/Normal-8371.png,True,False,299,299,L,111.51,333a5aac84f5083ed55879f899882efb
Normal-8372.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8372.png,Normal/images/Normal-8372.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8372.png,Normal/masks/Normal-8372.png,True,False,299,299,L,121.76,e69c4f68842d70327c80c0903b256302
Normal-8373.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8373.png,Normal/images/Normal-8373.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8373.png,Normal/masks/Normal-8373.png,True,False,299,299,L,128.75,242a3cf550749ad7817c63af6f43511a
Normal-8374.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8374.png,Normal/images/Normal-8374.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8374.png,Normal/masks/Normal-8374.png,True,False,299,299,L,131.77,17695f7803a0a0bda7133eed7940437f
Normal-8375.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8375.png,Normal/images/Normal-8375.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8375.png,Normal/masks/Normal-8375.png,True,False,299,299,L,154.41,7cd52c0459a0eec197559b47a6d314bc
Normal-8376.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8376.png,Normal/images/Normal-8376.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8376.png,Normal/masks/Normal-8376.png,True,False,299,299,L,128.25,02e4ec5f51e5bd9a4d8c5ab9cb8336c7
Normal-8377.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8377.png,Normal/images/Normal-8377.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8377.png,Normal/masks/Normal-8377.png,True,False,299,299,L,109.18,2ad117684bc56f96a735b6a886da71db
Normal-8378.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8378.png,Normal/images/Normal-8378.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8378.png,Normal/masks/Normal-8378.png,True,False,299,299,L,130.86,bdf691af1430514701eae45c5145aaf0
Normal-8379.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8379.png,Normal/images/Normal-8379.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8379.png,Normal/masks/Normal-8379.png,True,False,299,299,L,164.36,0d996e79b74ea5eee3baec9a92f36831
Normal-838.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-838.png,Normal/images/Normal-838.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-838.png,Normal/masks/Normal-838.png,True,False,299,299,L,138.02,f95c7df1ff3b033d730397b6169cba3f
Normal-8380.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8380.png,Normal/images/Normal-8380.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8380.png,Normal/masks/Normal-8380.png,True,False,299,299,L,106.61,520e1fd0ada10f284ea1ebe9dc43cf3c
Normal-8381.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8381.png,Normal/images/Normal-8381.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8381.png,Normal/masks/Normal-8381.png,True,False,299,299,L,101.21,bdf66e1a4afba3d3e13108842571ff6f
Normal-8382.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8382.png,Normal/images/Normal-8382.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8382.png,Normal/masks/Normal-8382.png,True,False,299,299,L,122.1,2e6fb069124a2e06fa9e4bf1f39e5098
Normal-8383.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8383.png,Normal/images/Normal-8383.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8383.png,Normal/masks/Normal-8383.png,True,False,299,299,L,140.69,fd6490e41a3c60926d3d446b82642a13
Normal-8384.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8384.png,Normal/images/Normal-8384.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8384.png,Normal/masks/Normal-8384.png,True,False,299,299,L,124.44,90b2e321f4a96a2662fd97400e1a6564
Normal-8385.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8385.png,Normal/images/Normal-8385.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8385.png,Normal/masks/Normal-8385.png,True,False,299,299,L,118.8,a366b7c46f8611f34397d8355adbf37c
Normal-8386.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8386.png,Normal/images/Normal-8386.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8386.png,Normal/masks/Normal-8386.png,True,False,299,299,L,107.36,33dc653f4f669269dc6c8e75e18ff721
Normal-8387.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8387.png,Normal/images/Normal-8387.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8387.png,Normal/masks/Normal-8387.png,True,False,299,299,L,99.02,f72d3dfaef6b8b4e2a0a5a21027be315
Normal-8388.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8388.png,Normal/images/Normal-8388.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8388.png,Normal/masks/Normal-8388.png,True,False,299,299,L,108.57,90a2e5056bc32cddae6eb9b8947f316d
Normal-8389.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8389.png,Normal/images/Normal-8389.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8389.png,Normal/masks/Normal-8389.png,True,False,299,299,L,160.92,bddaca7a9b91ef856a16dc1a6c7e8537
Normal-839.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-839.png,Normal/images/Normal-839.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-839.png,Normal/masks/Normal-839.png,True,False,299,299,L,126.48,725ed73d9b2e949ff665ca468b1ad725
Normal-8390.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8390.png,Normal/images/Normal-8390.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8390.png,Normal/masks/Normal-8390.png,True,False,299,299,L,158.31,d48833224c26cefc76f1ab3dd1b564c2
Normal-8391.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8391.png,Normal/images/Normal-8391.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8391.png,Normal/masks/Normal-8391.png,True,False,299,299,L,173.0,86e598b669ba04ab9124aad91766881c
Normal-8392.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8392.png,Normal/images/Normal-8392.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8392.png,Normal/masks/Normal-8392.png,True,False,299,299,L,136.09,874fa017cdd0cc79921dbb979bf498ed
Normal-8393.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8393.png,Normal/images/Normal-8393.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8393.png,Normal/masks/Normal-8393.png,True,False,299,299,L,120.84,c06a4a2a214e442c95268d4dedb9ba3d
Normal-8394.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8394.png,Normal/images/Normal-8394.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8394.png,Normal/masks/Normal-8394.png,True,False,299,299,L,105.26,c32efd2a9a9483d6aa77cd0e8dd90604
Normal-8395.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8395.png,Normal/images/Normal-8395.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8395.png,Normal/masks/Normal-8395.png,True,False,299,299,L,127.2,e13c5dc85b3c27114fe0edaa0427ef8e
Normal-8396.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8396.png,Normal/images/Normal-8396.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8396.png,Normal/masks/Normal-8396.png,True,False,299,299,L,116.69,72f635f3abdc3d9a4aac0c59abd6ada4
Normal-8397.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8397.png,Normal/images/Normal-8397.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8397.png,Normal/masks/Normal-8397.png,True,False,299,299,L,172.52,73ecc4c54ed5ee4593b396e617d24e8a
Normal-8398.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8398.png,Normal/images/Normal-8398.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8398.png,Normal/masks/Normal-8398.png,True,False,299,299,L,114.78,d39213cf0fee0d01c3e3f3778a8fc67f
Normal-8399.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8399.png,Normal/images/Normal-8399.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8399.png,Normal/masks/Normal-8399.png,True,False,299,299,L,136.46,5d399046f779b4c4fcc22cb91649988a
Normal-84.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-84.png,Normal/images/Normal-84.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-84.png,Normal/masks/Normal-84.png,True,False,299,299,L,112.36,240a87c362e4a850a328e4fa730d86de
Normal-840.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-840.png,Normal/images/Normal-840.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-840.png,Normal/masks/Normal-840.png,True,False,299,299,L,109.8,d5ca83d33913d957c9aeccb627f4c55d
Normal-8400.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8400.png,Normal/images/Normal-8400.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8400.png,Normal/masks/Normal-8400.png,True,False,299,299,L,181.35,e3502fd3cef3a7403831dc72d24165de
Normal-8401.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8401.png,Normal/images/Normal-8401.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8401.png,Normal/masks/Normal-8401.png,True,False,299,299,L,117.6,3d30ffa78d87bc6f315535c95f3b1b9a
Normal-8402.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8402.png,Normal/images/Normal-8402.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8402.png,Normal/masks/Normal-8402.png,True,False,299,299,L,136.76,f7bd042cd0abd22542f6b4d824e5f491
Normal-8403.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8403.png,Normal/images/Normal-8403.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8403.png,Normal/masks/Normal-8403.png,True,False,299,299,L,138.76,bd7480c1aeaebc0d6477b9e4ed27f800
Normal-8404.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8404.png,Normal/images/Normal-8404.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8404.png,Normal/masks/Normal-8404.png,True,False,299,299,L,114.02,5bcbedc0b52c34f6ff74f89357db4059
Normal-8405.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8405.png,Normal/images/Normal-8405.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8405.png,Normal/masks/Normal-8405.png,True,False,299,299,L,158.8,fee8f592a672d0f95550051ce249065f
Normal-8406.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8406.png,Normal/images/Normal-8406.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8406.png,Normal/masks/Normal-8406.png,True,False,299,299,L,98.03,79f32e5b18ce2d0a085074d4ddf3a0a9
Normal-8407.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8407.png,Normal/images/Normal-8407.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8407.png,Normal/masks/Normal-8407.png,True,False,299,299,L,126.32,074966399b51e419ae5ad56e41a56e41
Normal-8408.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8408.png,Normal/images/Normal-8408.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8408.png,Normal/masks/Normal-8408.png,True,False,299,299,L,115.8,9b4028af0f70143822398044ae87b0b3
Normal-8409.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8409.png,Normal/images/Normal-8409.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8409.png,Normal/masks/Normal-8409.png,True,False,299,299,L,108.67,57133c8ff18c1fc142732bec8a41892a
Normal-841.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-841.png,Normal/images/Normal-841.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-841.png,Normal/masks/Normal-841.png,True,False,299,299,L,110.97,05d0f670d650283614f7270ca90e11d5
Normal-8410.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8410.png,Normal/images/Normal-8410.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8410.png,Normal/masks/Normal-8410.png,True,False,299,299,L,164.93,a490e64abffdd29620919ffefdfda3e3
Normal-8411.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8411.png,Normal/images/Normal-8411.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8411.png,Normal/masks/Normal-8411.png,True,False,299,299,L,119.56,b945cdd2fb52f8001e9202d7c646c1b3
Normal-8412.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8412.png,Normal/images/Normal-8412.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8412.png,Normal/masks/Normal-8412.png,True,False,299,299,L,123.14,56e8469e46eba05ae4efa947c3352876
Normal-8413.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8413.png,Normal/images/Normal-8413.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8413.png,Normal/masks/Normal-8413.png,True,False,299,299,L,161.82,65eb9c289ca4af4d30ae5f6086ec08e7
Normal-8414.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8414.png,Normal/images/Normal-8414.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8414.png,Normal/masks/Normal-8414.png,True,False,299,299,L,116.6,6527ca1d24367a198fdfa5df97711bd6
Normal-8415.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8415.png,Normal/images/Normal-8415.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8415.png,Normal/masks/Normal-8415.png,True,False,299,299,L,122.07,ae1b4a8bc275f57d085a34aa2f33b2f4
Normal-8416.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8416.png,Normal/images/Normal-8416.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8416.png,Normal/masks/Normal-8416.png,True,False,299,299,L,127.61,d581bfa6d470882961a9d792398d6073
Normal-8417.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8417.png,Normal/images/Normal-8417.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8417.png,Normal/masks/Normal-8417.png,True,False,299,299,L,136.18,e497cf24d3af9481db1b9d879f37d435
Normal-8418.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8418.png,Normal/images/Normal-8418.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8418.png,Normal/masks/Normal-8418.png,True,False,299,299,L,184.45,e075b0e9f2ab6d488c896bdd2fa1d551
Normal-8419.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8419.png,Normal/images/Normal-8419.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8419.png,Normal/masks/Normal-8419.png,True,False,299,299,L,122.41,e2d897881448c5008b38524995edc48e
Normal-842.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-842.png,Normal/images/Normal-842.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-842.png,Normal/masks/Normal-842.png,True,False,299,299,L,114.51,d89ca5c49a326295cb8fcdedf29cbefc
Normal-8420.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8420.png,Normal/images/Normal-8420.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8420.png,Normal/masks/Normal-8420.png,True,False,299,299,L,114.13,cc120d91bcb4e15c0e24cca7a944cfc3
Normal-8421.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8421.png,Normal/images/Normal-8421.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8421.png,Normal/masks/Normal-8421.png,True,False,299,299,L,139.96,acf5954152a484404425e6fcbf4f65d5
Normal-8422.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8422.png,Normal/images/Normal-8422.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8422.png,Normal/masks/Normal-8422.png,True,False,299,299,L,121.29,3b2b6e11cf9005c992446fd114a0195a
Normal-8423.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8423.png,Normal/images/Normal-8423.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8423.png,Normal/masks/Normal-8423.png,True,False,299,299,L,118.09,df3adc389850775e396129b05b9c7551
Normal-8424.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8424.png,Normal/images/Normal-8424.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8424.png,Normal/masks/Normal-8424.png,True,False,299,299,L,138.5,510b250834ffc6fa3746cded9329b0f3
Normal-8425.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8425.png,Normal/images/Normal-8425.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8425.png,Normal/masks/Normal-8425.png,True,False,299,299,L,122.24,0280708c46ce047747337772c6f611e0
Normal-8426.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8426.png,Normal/images/Normal-8426.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8426.png,Normal/masks/Normal-8426.png,True,False,299,299,L,93.7,e500ac0fd1ce1942d8f8776a28450fa9
Normal-8427.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8427.png,Normal/images/Normal-8427.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8427.png,Normal/masks/Normal-8427.png,True,False,299,299,L,104.75,4a0070d0f00c861819aa8bd6a94a28ea
Normal-8428.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8428.png,Normal/images/Normal-8428.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8428.png,Normal/masks/Normal-8428.png,True,False,299,299,L,161.88,3dca034c6ee9b232e70ea9185dcdb72d
Normal-8429.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8429.png,Normal/images/Normal-8429.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8429.png,Normal/masks/Normal-8429.png,True,False,299,299,L,117.3,a2ea820cdaeec3067ed690fdb2b10e08
Normal-843.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-843.png,Normal/images/Normal-843.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-843.png,Normal/masks/Normal-843.png,True,False,299,299,L,115.13,1a9345c6deec5762763d3852d7c7be39
Normal-8430.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8430.png,Normal/images/Normal-8430.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8430.png,Normal/masks/Normal-8430.png,True,False,299,299,L,111.25,22edfe7d15ff6a6df3bdeb5c930972a8
Normal-8431.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8431.png,Normal/images/Normal-8431.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8431.png,Normal/masks/Normal-8431.png,True,False,299,299,L,161.69,b2e22ff6a69cba9029059cf56974f800
Normal-8432.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8432.png,Normal/images/Normal-8432.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8432.png,Normal/masks/Normal-8432.png,True,False,299,299,L,132.33,d8e29bb7c2055e9e5d070258764445c2
Normal-8433.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8433.png,Normal/images/Normal-8433.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8433.png,Normal/masks/Normal-8433.png,True,False,299,299,L,148.44,4b90c7804a777a005232390289ae8ca7
Normal-8434.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8434.png,Normal/images/Normal-8434.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8434.png,Normal/masks/Normal-8434.png,True,False,299,299,L,152.79,a06b46dc54faba56b470d0fc6668a70b
Normal-8435.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8435.png,Normal/images/Normal-8435.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8435.png,Normal/masks/Normal-8435.png,True,False,299,299,L,183.28,531ab64c865676e9333fb1e56d4ec7f7
Normal-8436.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8436.png,Normal/images/Normal-8436.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8436.png,Normal/masks/Normal-8436.png,True,False,299,299,L,150.61,8de2e5251973e4c804cba9d99025de7f
Normal-8437.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8437.png,Normal/images/Normal-8437.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8437.png,Normal/masks/Normal-8437.png,True,False,299,299,L,110.18,bdbd6daebf363bfc90e77314488a6eb9
Normal-8438.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8438.png,Normal/images/Normal-8438.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8438.png,Normal/masks/Normal-8438.png,True,False,299,299,L,134.22,c511db2592e888df3d816410a60c68c9
Normal-8439.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8439.png,Normal/images/Normal-8439.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8439.png,Normal/masks/Normal-8439.png,True,False,299,299,L,136.79,e2167df5eca86416683eba0ebcdd85b7
Normal-844.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-844.png,Normal/images/Normal-844.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-844.png,Normal/masks/Normal-844.png,True,False,299,299,L,122.13,16f75b4e4d6497f84e816eda6049d884
Normal-8440.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8440.png,Normal/images/Normal-8440.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8440.png,Normal/masks/Normal-8440.png,True,False,299,299,L,123.71,0c581d3123e932e6742ddf46835f41c9
Normal-8441.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8441.png,Normal/images/Normal-8441.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8441.png,Normal/masks/Normal-8441.png,True,False,299,299,L,122.52,5556172adf0a007ce49eac5c6c12f3c9
Normal-8442.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8442.png,Normal/images/Normal-8442.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8442.png,Normal/masks/Normal-8442.png,True,False,299,299,L,125.97,8cbd9c2b296204999dd97eb0cea0cd82
Normal-8443.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8443.png,Normal/images/Normal-8443.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8443.png,Normal/masks/Normal-8443.png,True,False,299,299,L,163.92,fd469e09def9631204fc38f4f593b7b3
Normal-8444.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8444.png,Normal/images/Normal-8444.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8444.png,Normal/masks/Normal-8444.png,True,False,299,299,L,130.64,941d7245b15c0e8357033ead1084d9f6
Normal-8445.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8445.png,Normal/images/Normal-8445.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8445.png,Normal/masks/Normal-8445.png,True,False,299,299,L,125.87,fba110e16f0c0a21abab76add69b7175
Normal-8446.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8446.png,Normal/images/Normal-8446.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8446.png,Normal/masks/Normal-8446.png,True,False,299,299,L,130.86,ed133c35cc0753c04c4cfbc0e599fd0e
Normal-8447.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8447.png,Normal/images/Normal-8447.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8447.png,Normal/masks/Normal-8447.png,True,False,299,299,L,120.38,192b00327b90372d815fb97153b9735c
Normal-8448.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8448.png,Normal/images/Normal-8448.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8448.png,Normal/masks/Normal-8448.png,True,False,299,299,L,151.58,cffa553ac3051d80ba3caca5d9c95ae7
Normal-8449.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8449.png,Normal/images/Normal-8449.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8449.png,Normal/masks/Normal-8449.png,True,False,299,299,L,179.45,fe3dd90a95c6115eb824297d96c9fe17
Normal-845.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-845.png,Normal/images/Normal-845.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-845.png,Normal/masks/Normal-845.png,True,False,299,299,L,119.18,daf3e03842f70761657b6a6a8cf4f0e7
Normal-8450.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8450.png,Normal/images/Normal-8450.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8450.png,Normal/masks/Normal-8450.png,True,False,299,299,L,112.01,0c9e2bf68ebdedf77863211cd5be6048
Normal-8451.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8451.png,Normal/images/Normal-8451.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8451.png,Normal/masks/Normal-8451.png,True,False,299,299,L,108.26,ee620b0d3113f17232679849b805163a
Normal-8452.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8452.png,Normal/images/Normal-8452.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8452.png,Normal/masks/Normal-8452.png,True,False,299,299,L,168.01,11b914718d60358914c8e76fffd024a6
Normal-8453.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8453.png,Normal/images/Normal-8453.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8453.png,Normal/masks/Normal-8453.png,True,False,299,299,L,111.39,bf757cacacb1f99a408a26b4cb1e4528
Normal-8454.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8454.png,Normal/images/Normal-8454.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8454.png,Normal/masks/Normal-8454.png,True,False,299,299,L,157.21,ecbce65d10cb2000b9c7742540dd4536
Normal-8455.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8455.png,Normal/images/Normal-8455.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8455.png,Normal/masks/Normal-8455.png,True,False,299,299,L,102.84,03368e8f6e81eb582f908cbebe14168d
Normal-8456.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8456.png,Normal/images/Normal-8456.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8456.png,Normal/masks/Normal-8456.png,True,False,299,299,L,149.77,b7905fc8d7c6a03664651c8e7c57229d
Normal-8457.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8457.png,Normal/images/Normal-8457.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8457.png,Normal/masks/Normal-8457.png,True,False,299,299,L,175.82,f92cd16d23b53e2b468ae090b3ee2971
Normal-8458.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8458.png,Normal/images/Normal-8458.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8458.png,Normal/masks/Normal-8458.png,True,False,299,299,L,91.01,e83fe6ffa95b04cb278175008e1ebe95
Normal-8459.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8459.png,Normal/images/Normal-8459.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8459.png,Normal/masks/Normal-8459.png,True,False,299,299,L,115.34,39ecc0c6990087845345d2d6eb568a3f
Normal-846.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-846.png,Normal/images/Normal-846.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-846.png,Normal/masks/Normal-846.png,True,False,299,299,L,106.51,c05850bd735180d9d7ab26413808e844
Normal-8460.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8460.png,Normal/images/Normal-8460.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8460.png,Normal/masks/Normal-8460.png,True,False,299,299,L,110.97,c3b11ef7f24f9194ed838cda1667f827
Normal-8461.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8461.png,Normal/images/Normal-8461.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8461.png,Normal/masks/Normal-8461.png,True,False,299,299,L,124.14,e766588c21d713daf91d1b7b6341ffd5
Normal-8462.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8462.png,Normal/images/Normal-8462.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8462.png,Normal/masks/Normal-8462.png,True,False,299,299,L,162.93,3029808508234bee2f724d7d1c1c3744
Normal-8463.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8463.png,Normal/images/Normal-8463.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8463.png,Normal/masks/Normal-8463.png,True,False,299,299,L,127.96,ca8141ba5e18a653dac561ee1a198ab8
Normal-8464.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8464.png,Normal/images/Normal-8464.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8464.png,Normal/masks/Normal-8464.png,True,False,299,299,L,106.13,02648a1fd3dfe29dfd929dd6dd80af55
Normal-8465.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8465.png,Normal/images/Normal-8465.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8465.png,Normal/masks/Normal-8465.png,True,False,299,299,L,157.51,5693cb91e055c735c6a735ae29be3fcb
Normal-8466.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8466.png,Normal/images/Normal-8466.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8466.png,Normal/masks/Normal-8466.png,True,False,299,299,L,186.23,743e3e5f395690b8746b962500fe9a05
Normal-8467.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8467.png,Normal/images/Normal-8467.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8467.png,Normal/masks/Normal-8467.png,True,False,299,299,L,125.0,65f6f7c2e23d748703c57ca341742264
Normal-8468.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8468.png,Normal/images/Normal-8468.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8468.png,Normal/masks/Normal-8468.png,True,False,299,299,L,113.36,49902898bd715575e51354b4effff1d5
Normal-8469.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8469.png,Normal/images/Normal-8469.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8469.png,Normal/masks/Normal-8469.png,True,False,299,299,L,144.85,1cede3b57184ab999006a1838aabab49
Normal-847.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-847.png,Normal/images/Normal-847.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-847.png,Normal/masks/Normal-847.png,True,False,299,299,L,128.13,4de069d3872e27df879aec625ad2136b
Normal-8470.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8470.png,Normal/images/Normal-8470.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8470.png,Normal/masks/Normal-8470.png,True,False,299,299,L,160.84,e59af554f76305a6b7f590ad4fc0c3c0
Normal-8471.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8471.png,Normal/images/Normal-8471.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8471.png,Normal/masks/Normal-8471.png,True,False,299,299,L,120.59,f93b63d33aa74ef36fb624b7ed311d4e
Normal-8472.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8472.png,Normal/images/Normal-8472.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8472.png,Normal/masks/Normal-8472.png,True,False,299,299,L,140.57,8e5ec562b9cc285ea9e460b8ebb5be0d
Normal-8473.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8473.png,Normal/images/Normal-8473.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8473.png,Normal/masks/Normal-8473.png,True,False,299,299,L,157.58,9e3afa50a808da89731b2c9351a22e89
Normal-8474.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8474.png,Normal/images/Normal-8474.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8474.png,Normal/masks/Normal-8474.png,True,False,299,299,L,137.6,34432b282daf34b1dfccb33a6b4769dc
Normal-8475.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8475.png,Normal/images/Normal-8475.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8475.png,Normal/masks/Normal-8475.png,True,False,299,299,L,130.37,83a088999831bce2d20df9d68f841561
Normal-8476.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8476.png,Normal/images/Normal-8476.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8476.png,Normal/masks/Normal-8476.png,True,False,299,299,L,111.36,8a80b342ee6f5514c269e9ba17de1e88
Normal-8477.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8477.png,Normal/images/Normal-8477.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8477.png,Normal/masks/Normal-8477.png,True,False,299,299,L,131.67,5a684fedd8485144a5bc8732e283ff5e
Normal-8478.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8478.png,Normal/images/Normal-8478.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8478.png,Normal/masks/Normal-8478.png,True,False,299,299,L,122.57,e82ae56148faf0e3c4957efd29a52b88
Normal-8479.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8479.png,Normal/images/Normal-8479.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8479.png,Normal/masks/Normal-8479.png,True,False,299,299,L,118.01,1accac36195c13f8f1235b476b84bc8e
Normal-848.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-848.png,Normal/images/Normal-848.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-848.png,Normal/masks/Normal-848.png,True,False,299,299,L,132.21,ed18aac580b9a8c931d919d6ad7fc238
Normal-8480.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8480.png,Normal/images/Normal-8480.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8480.png,Normal/masks/Normal-8480.png,True,False,299,299,L,120.93,b54f0782a60afe0ad063443e81916d49
Normal-8481.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8481.png,Normal/images/Normal-8481.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8481.png,Normal/masks/Normal-8481.png,True,False,299,299,L,92.86,f06770784d188f25f3dabf115b4d1dad
Normal-8482.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8482.png,Normal/images/Normal-8482.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8482.png,Normal/masks/Normal-8482.png,True,False,299,299,L,106.16,a6d9e1b085792da21df509bd666f50cc
Normal-8483.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8483.png,Normal/images/Normal-8483.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8483.png,Normal/masks/Normal-8483.png,True,False,299,299,L,115.75,986d567106b4271a9e96f3d5fe5bca1a
Normal-8484.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8484.png,Normal/images/Normal-8484.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8484.png,Normal/masks/Normal-8484.png,True,False,299,299,L,171.71,b782d204bd18b164123c912b5f2399f5
Normal-8485.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8485.png,Normal/images/Normal-8485.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8485.png,Normal/masks/Normal-8485.png,True,False,299,299,L,118.78,9c006f4b816de8974303a54e0a273884
Normal-8486.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8486.png,Normal/images/Normal-8486.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8486.png,Normal/masks/Normal-8486.png,True,False,299,299,L,115.26,e2148741e7ee259de3de428814161f27
Normal-8487.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8487.png,Normal/images/Normal-8487.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8487.png,Normal/masks/Normal-8487.png,True,False,299,299,L,117.49,f0bca1c0ba36ca780b15e848062c2abe
Normal-8488.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8488.png,Normal/images/Normal-8488.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8488.png,Normal/masks/Normal-8488.png,True,False,299,299,L,176.4,cdbf3a32d46ee942a0c5dea72faad58e
Normal-8489.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8489.png,Normal/images/Normal-8489.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8489.png,Normal/masks/Normal-8489.png,True,False,299,299,L,109.89,cec500cbaffe64723b3833129173f623
Normal-849.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-849.png,Normal/images/Normal-849.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-849.png,Normal/masks/Normal-849.png,True,False,299,299,L,136.25,d670ad3c9e1f3e463d8022f983427648
Normal-8490.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8490.png,Normal/images/Normal-8490.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8490.png,Normal/masks/Normal-8490.png,True,False,299,299,L,135.82,9ebef9fcc0a84e8d5956bf6a951d238e
Normal-8491.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8491.png,Normal/images/Normal-8491.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8491.png,Normal/masks/Normal-8491.png,True,False,299,299,L,159.93,bfb0300190c5330ce853444de36ba24a
Normal-8492.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8492.png,Normal/images/Normal-8492.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8492.png,Normal/masks/Normal-8492.png,True,False,299,299,L,118.88,3a98ff6a3f9c4321a69795e6b8e6ead4
Normal-8493.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8493.png,Normal/images/Normal-8493.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8493.png,Normal/masks/Normal-8493.png,True,False,299,299,L,129.62,d4f24684bc12c1c19aa11d8c5bd9e5c4
Normal-8494.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8494.png,Normal/images/Normal-8494.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8494.png,Normal/masks/Normal-8494.png,True,False,299,299,L,121.71,c1b971c8bc0e515f0c3362d798d492ba
Normal-8495.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8495.png,Normal/images/Normal-8495.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8495.png,Normal/masks/Normal-8495.png,True,False,299,299,L,113.53,38d8658be119dcb37fa5c1717b90a680
Normal-8496.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8496.png,Normal/images/Normal-8496.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8496.png,Normal/masks/Normal-8496.png,True,False,299,299,L,125.04,3f724dcfeb1f39fe07e5f5e4e5f76031
Normal-8497.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8497.png,Normal/images/Normal-8497.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8497.png,Normal/masks/Normal-8497.png,True,False,299,299,L,131.07,90ccb8d0c80fc4de4182dec939af507c
Normal-8498.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8498.png,Normal/images/Normal-8498.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8498.png,Normal/masks/Normal-8498.png,True,False,299,299,L,126.35,01a7f3edd7419f12213be0faf4e4e353
Normal-8499.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8499.png,Normal/images/Normal-8499.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8499.png,Normal/masks/Normal-8499.png,True,False,299,299,L,139.4,7b6dd94560f717641b795527db44d9f8
Normal-85.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-85.png,Normal/images/Normal-85.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-85.png,Normal/masks/Normal-85.png,True,False,299,299,L,105.57,4b7ee83d0118c410bd69f1f01b990862
Normal-850.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-850.png,Normal/images/Normal-850.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-850.png,Normal/masks/Normal-850.png,True,False,299,299,L,118.15,099b69b5a1c15b72257377528a77ecd6
Normal-8500.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8500.png,Normal/images/Normal-8500.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8500.png,Normal/masks/Normal-8500.png,True,False,299,299,L,106.08,5448c195153fc95f43699bf8fee319e1
Normal-8501.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8501.png,Normal/images/Normal-8501.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8501.png,Normal/masks/Normal-8501.png,True,False,299,299,L,132.47,4b285463bad89a0951d38ca86bade826
Normal-8502.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8502.png,Normal/images/Normal-8502.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8502.png,Normal/masks/Normal-8502.png,True,False,299,299,L,150.18,6962e2d27ca3dd86dffabc051ffb24ef
Normal-8503.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8503.png,Normal/images/Normal-8503.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8503.png,Normal/masks/Normal-8503.png,True,False,299,299,L,117.46,4080fc412f7eeb53d2304d0cfd4a3d6e
Normal-8504.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8504.png,Normal/images/Normal-8504.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8504.png,Normal/masks/Normal-8504.png,True,False,299,299,L,128.9,0b750a1e1fec3d3b7a72fe7f38a189ee
Normal-8505.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8505.png,Normal/images/Normal-8505.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8505.png,Normal/masks/Normal-8505.png,True,False,299,299,L,173.51,c9bc16465bff7c16ca5b475d895e8617
Normal-8506.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8506.png,Normal/images/Normal-8506.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8506.png,Normal/masks/Normal-8506.png,True,False,299,299,L,110.3,8678f7dc2b166f66b951ef97428c1883
Normal-8507.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8507.png,Normal/images/Normal-8507.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8507.png,Normal/masks/Normal-8507.png,True,False,299,299,L,139.74,2dd106352a669ff940a77450664d23d7
Normal-8508.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8508.png,Normal/images/Normal-8508.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8508.png,Normal/masks/Normal-8508.png,True,False,299,299,L,143.03,1e0c994cd4de6ec07ed8fa3cdaa1f8dd
Normal-8509.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8509.png,Normal/images/Normal-8509.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8509.png,Normal/masks/Normal-8509.png,True,False,299,299,L,151.43,1391581f933d06f1450bc0ba362c323d
Normal-851.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-851.png,Normal/images/Normal-851.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-851.png,Normal/masks/Normal-851.png,True,False,299,299,L,124.49,b3a8b446cbc947577fbd9ea9d2f2f650
Normal-8510.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8510.png,Normal/images/Normal-8510.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8510.png,Normal/masks/Normal-8510.png,True,False,299,299,L,107.43,7a1aaad12e7357721f78c555a4b246dd
Normal-8511.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8511.png,Normal/images/Normal-8511.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8511.png,Normal/masks/Normal-8511.png,True,False,299,299,L,124.61,c98a28010e8f4d86ad2aa1bc0e37d7cf
Normal-8512.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8512.png,Normal/images/Normal-8512.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8512.png,Normal/masks/Normal-8512.png,True,False,299,299,L,113.92,b444cb7aa0f35ed26f21e1e1ea6ca5c1
Normal-8513.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8513.png,Normal/images/Normal-8513.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8513.png,Normal/masks/Normal-8513.png,True,False,299,299,L,145.92,41d59dd06e6cf3d31abcf2b6e44fa682
Normal-8514.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8514.png,Normal/images/Normal-8514.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8514.png,Normal/masks/Normal-8514.png,True,False,299,299,L,143.55,16c88216906dd2a9e81c419434c799ea
Normal-8515.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8515.png,Normal/images/Normal-8515.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8515.png,Normal/masks/Normal-8515.png,True,False,299,299,L,116.69,db78278798a509bd3f63d8fb573f7a83
Normal-8516.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8516.png,Normal/images/Normal-8516.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8516.png,Normal/masks/Normal-8516.png,True,False,299,299,L,139.22,4ce2baac0615c5f08ab43ff310563cb2
Normal-8517.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8517.png,Normal/images/Normal-8517.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8517.png,Normal/masks/Normal-8517.png,True,False,299,299,L,138.25,95a81ce8a3b92bdcad8717e1cce92ec5
Normal-8518.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8518.png,Normal/images/Normal-8518.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8518.png,Normal/masks/Normal-8518.png,True,False,299,299,L,149.4,7fb5ed18f2c94a7514a7a6f3a324b563
Normal-8519.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8519.png,Normal/images/Normal-8519.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8519.png,Normal/masks/Normal-8519.png,True,False,299,299,L,117.21,01bc747438d10c66e34ec12b773c86b2
Normal-852.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-852.png,Normal/images/Normal-852.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-852.png,Normal/masks/Normal-852.png,True,False,299,299,L,110.59,6ddb46c60c2a7b4e7bba25227a0abe58
Normal-8520.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8520.png,Normal/images/Normal-8520.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8520.png,Normal/masks/Normal-8520.png,True,False,299,299,L,162.28,16e9238e107aaa75e812a0ae0bdd6fa9
Normal-8521.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8521.png,Normal/images/Normal-8521.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8521.png,Normal/masks/Normal-8521.png,True,False,299,299,L,149.5,b8c56821000cd6fda33d032046a99913
Normal-8522.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8522.png,Normal/images/Normal-8522.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8522.png,Normal/masks/Normal-8522.png,True,False,299,299,L,147.24,d144b1b0cb00e82ea2f84321f11b8ce9
Normal-8523.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8523.png,Normal/images/Normal-8523.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8523.png,Normal/masks/Normal-8523.png,True,False,299,299,L,100.17,490ca4f6a0334a56d5c8c966122ec673
Normal-8524.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8524.png,Normal/images/Normal-8524.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8524.png,Normal/masks/Normal-8524.png,True,False,299,299,L,162.56,23a0c51905bce4f452dc2c5d4597cd5f
Normal-8525.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8525.png,Normal/images/Normal-8525.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8525.png,Normal/masks/Normal-8525.png,True,False,299,299,L,137.97,ba6257217e5213375f49da3ebe723a28
Normal-8526.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8526.png,Normal/images/Normal-8526.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8526.png,Normal/masks/Normal-8526.png,True,False,299,299,L,103.78,612311a6e94d792d9b03087188170a68
Normal-8527.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8527.png,Normal/images/Normal-8527.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8527.png,Normal/masks/Normal-8527.png,True,False,299,299,L,156.99,093c2ab62609ed6a767e212a8d37f164
Normal-8528.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8528.png,Normal/images/Normal-8528.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8528.png,Normal/masks/Normal-8528.png,True,False,299,299,L,121.78,dd13f54b903116640538ac89fee56fca
Normal-8529.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8529.png,Normal/images/Normal-8529.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8529.png,Normal/masks/Normal-8529.png,True,False,299,299,L,120.22,78affb0a865a74aa22da08f711c1da56
Normal-853.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-853.png,Normal/images/Normal-853.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-853.png,Normal/masks/Normal-853.png,True,False,299,299,L,129.52,0a8104de709bf50408faef35de186138
Normal-8530.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8530.png,Normal/images/Normal-8530.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8530.png,Normal/masks/Normal-8530.png,True,False,299,299,L,102.23,19c0868fd3cf1525865d2b1310be8a6b
Normal-8531.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8531.png,Normal/images/Normal-8531.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8531.png,Normal/masks/Normal-8531.png,True,False,299,299,L,108.29,3b0b64533ab6dd22c4094962b704a9b4
Normal-8532.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8532.png,Normal/images/Normal-8532.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8532.png,Normal/masks/Normal-8532.png,True,False,299,299,L,111.08,c7e9358648e56a1a76bae3194cf95055
Normal-8533.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8533.png,Normal/images/Normal-8533.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8533.png,Normal/masks/Normal-8533.png,True,False,299,299,L,143.42,63709014fc8fa47d087afdd27c3439a9
Normal-8534.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8534.png,Normal/images/Normal-8534.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8534.png,Normal/masks/Normal-8534.png,True,False,299,299,L,121.6,a2625519435b9f9c3aac922fce3ccfb3
Normal-8535.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8535.png,Normal/images/Normal-8535.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8535.png,Normal/masks/Normal-8535.png,True,False,299,299,L,143.66,73e7bbf20cc990e9f5f97918b8e2218d
Normal-8536.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8536.png,Normal/images/Normal-8536.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8536.png,Normal/masks/Normal-8536.png,True,False,299,299,L,148.16,c1233097d291b42c025c57f7833a53ba
Normal-8537.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8537.png,Normal/images/Normal-8537.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8537.png,Normal/masks/Normal-8537.png,True,False,299,299,L,114.31,7a7e72fce36a1fccf3f8c6da1fe65b28
Normal-8538.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8538.png,Normal/images/Normal-8538.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8538.png,Normal/masks/Normal-8538.png,True,False,299,299,L,98.9,8ad44e636e13f5bd138debde0cb376ff
Normal-8539.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8539.png,Normal/images/Normal-8539.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8539.png,Normal/masks/Normal-8539.png,True,False,299,299,L,171.28,4aa410cad6e69545c61248cb768fb99b
Normal-854.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-854.png,Normal/images/Normal-854.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-854.png,Normal/masks/Normal-854.png,True,False,299,299,L,123.08,6d2abbb99bbf2c62a8e2168a63a9ac4c
Normal-8540.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8540.png,Normal/images/Normal-8540.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8540.png,Normal/masks/Normal-8540.png,True,False,299,299,L,77.3,dedcd292454464f2239486aabe6769d3
Normal-8541.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8541.png,Normal/images/Normal-8541.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8541.png,Normal/masks/Normal-8541.png,True,False,299,299,L,104.01,840598e86e4f624dae3158882f4c7b4f
Normal-8542.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8542.png,Normal/images/Normal-8542.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8542.png,Normal/masks/Normal-8542.png,True,False,299,299,L,117.24,da405431a37967ba8517a578f19aaf39
Normal-8543.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8543.png,Normal/images/Normal-8543.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8543.png,Normal/masks/Normal-8543.png,True,False,299,299,L,117.11,ae2d5447837e07d76de8ef741d4d0f09
Normal-8544.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8544.png,Normal/images/Normal-8544.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8544.png,Normal/masks/Normal-8544.png,True,False,299,299,L,90.42,39a54bcedefb3761dba47adc4c21d56f
Normal-8545.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8545.png,Normal/images/Normal-8545.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8545.png,Normal/masks/Normal-8545.png,True,False,299,299,L,127.85,6d8ca917703fa8beb53d80ca4fae32ba
Normal-8546.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8546.png,Normal/images/Normal-8546.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8546.png,Normal/masks/Normal-8546.png,True,False,299,299,L,141.68,a07088435c1227bcb94f870b31b86a9f
Normal-8547.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8547.png,Normal/images/Normal-8547.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8547.png,Normal/masks/Normal-8547.png,True,False,299,299,L,96.38,b92404b9b5b8c872268c9fac6bf8dbf9
Normal-8548.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8548.png,Normal/images/Normal-8548.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8548.png,Normal/masks/Normal-8548.png,True,False,299,299,L,116.27,d39a87302d17906b7664ada880689239
Normal-8549.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8549.png,Normal/images/Normal-8549.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8549.png,Normal/masks/Normal-8549.png,True,False,299,299,L,128.05,97822b083a2b059dce533f4a7c06dbd1
Normal-855.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-855.png,Normal/images/Normal-855.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-855.png,Normal/masks/Normal-855.png,True,False,299,299,L,115.43,48b18400aa61216a63a7661e28f573da
Normal-8550.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8550.png,Normal/images/Normal-8550.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8550.png,Normal/masks/Normal-8550.png,True,False,299,299,L,126.53,8f48074be8e5a3a9ff0484b8ca2b219e
Normal-8551.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8551.png,Normal/images/Normal-8551.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8551.png,Normal/masks/Normal-8551.png,True,False,299,299,L,100.17,ddcd7f6a874ed674bb9d19d61ff3d823
Normal-8552.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8552.png,Normal/images/Normal-8552.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8552.png,Normal/masks/Normal-8552.png,True,False,299,299,L,129.5,face8da00c208b7efdb70e223ffba147
Normal-8553.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8553.png,Normal/images/Normal-8553.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8553.png,Normal/masks/Normal-8553.png,True,False,299,299,L,122.0,2c1d64f706768bc5047a97f791dd64fa
Normal-8554.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8554.png,Normal/images/Normal-8554.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8554.png,Normal/masks/Normal-8554.png,True,False,299,299,L,116.1,6829030cd9634fb6fccdc41cddee0a93
Normal-8555.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8555.png,Normal/images/Normal-8555.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8555.png,Normal/masks/Normal-8555.png,True,False,299,299,L,104.98,ae74d6ca65d5a69972bf0b44b8949f56
Normal-8556.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8556.png,Normal/images/Normal-8556.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8556.png,Normal/masks/Normal-8556.png,True,False,299,299,L,130.38,d541dd481c4c5a6f7a1b67115741316f
Normal-8557.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8557.png,Normal/images/Normal-8557.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8557.png,Normal/masks/Normal-8557.png,True,False,299,299,L,123.74,d65c286353444f01962f917f6c1ed0c1
Normal-8558.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8558.png,Normal/images/Normal-8558.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8558.png,Normal/masks/Normal-8558.png,True,False,299,299,L,93.49,91036de7bda9650a3d37515693d61ecc
Normal-8559.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8559.png,Normal/images/Normal-8559.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8559.png,Normal/masks/Normal-8559.png,True,False,299,299,L,108.28,55f455a794edcb206f3e4dff08e2acf0
Normal-856.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-856.png,Normal/images/Normal-856.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-856.png,Normal/masks/Normal-856.png,True,False,299,299,L,112.17,60d1031aabfd7bb78c7c6df5f9c8a310
Normal-8560.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8560.png,Normal/images/Normal-8560.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8560.png,Normal/masks/Normal-8560.png,True,False,299,299,L,125.5,0934b7785555279d68a73883e00ad30f
Normal-8561.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8561.png,Normal/images/Normal-8561.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8561.png,Normal/masks/Normal-8561.png,True,False,299,299,L,146.81,1c6d9fb3633904f471d8ef4a40d7f825
Normal-8562.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8562.png,Normal/images/Normal-8562.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8562.png,Normal/masks/Normal-8562.png,True,False,299,299,L,97.02,d5157a215111c25dcacbcd6b25d152bd
Normal-8563.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8563.png,Normal/images/Normal-8563.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8563.png,Normal/masks/Normal-8563.png,True,False,299,299,L,93.51,f47f3a07b6a5866fa25f7002d65ee154
Normal-8564.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8564.png,Normal/images/Normal-8564.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8564.png,Normal/masks/Normal-8564.png,True,False,299,299,L,88.88,89b4c0451cf6058adbbf25361d1b1268
Normal-8565.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8565.png,Normal/images/Normal-8565.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8565.png,Normal/masks/Normal-8565.png,True,False,299,299,L,147.21,b6aa5c23e857011dde9f8dbc267e3dfb
Normal-8566.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8566.png,Normal/images/Normal-8566.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8566.png,Normal/masks/Normal-8566.png,True,False,299,299,L,105.39,66b8f2a0dcf123092967e45248d25390
Normal-8567.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8567.png,Normal/images/Normal-8567.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8567.png,Normal/masks/Normal-8567.png,True,False,299,299,L,123.35,a18c79c8c1119df01d65b225e7fbe55a
Normal-8568.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8568.png,Normal/images/Normal-8568.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8568.png,Normal/masks/Normal-8568.png,True,False,299,299,L,107.43,7ef2ce2651d1854cdd2debb2bbc80a2f
Normal-8569.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8569.png,Normal/images/Normal-8569.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8569.png,Normal/masks/Normal-8569.png,True,False,299,299,L,119.72,2cf7deaa1cb2349afe3ca55bbbe9b61a
Normal-857.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-857.png,Normal/images/Normal-857.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-857.png,Normal/masks/Normal-857.png,True,False,299,299,L,113.53,0bdb26ea048b19581eb224a85eba91d2
Normal-8570.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8570.png,Normal/images/Normal-8570.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8570.png,Normal/masks/Normal-8570.png,True,False,299,299,L,169.04,302ec0006393d9a9c64f5d2266bd0b1c
Normal-8571.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8571.png,Normal/images/Normal-8571.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8571.png,Normal/masks/Normal-8571.png,True,False,299,299,L,109.81,2203c970ad7ab3fe28e049762443fa34
Normal-8572.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8572.png,Normal/images/Normal-8572.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8572.png,Normal/masks/Normal-8572.png,True,False,299,299,L,136.75,a2be2a63411549a5e1692ef36a49ba4c
Normal-8573.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8573.png,Normal/images/Normal-8573.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8573.png,Normal/masks/Normal-8573.png,True,False,299,299,L,97.71,34b82ec4aae86ea7014837c1dd41f72a
Normal-8574.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8574.png,Normal/images/Normal-8574.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8574.png,Normal/masks/Normal-8574.png,True,False,299,299,L,113.19,4f87a61e60ef33ba226da90f598db0d3
Normal-8575.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8575.png,Normal/images/Normal-8575.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8575.png,Normal/masks/Normal-8575.png,True,False,299,299,L,131.47,617097200cf52dd28f5d78fd4a9173a6
Normal-8576.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8576.png,Normal/images/Normal-8576.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8576.png,Normal/masks/Normal-8576.png,True,False,299,299,L,111.26,6aeba2fa981f8284c7e1dc3abad6af67
Normal-8577.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8577.png,Normal/images/Normal-8577.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8577.png,Normal/masks/Normal-8577.png,True,False,299,299,L,120.81,7576b1d16e442ec6a5e76378553d1d29
Normal-8578.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8578.png,Normal/images/Normal-8578.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8578.png,Normal/masks/Normal-8578.png,True,False,299,299,L,112.03,ceab284354c0e8df67c098fce7383d02
Normal-8579.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8579.png,Normal/images/Normal-8579.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8579.png,Normal/masks/Normal-8579.png,True,False,299,299,L,120.12,0eafa298d001090ec40d4b3895e7aa93
Normal-858.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-858.png,Normal/images/Normal-858.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-858.png,Normal/masks/Normal-858.png,True,False,299,299,L,148.69,379a69827180996cdb0224a839feeefd
Normal-8580.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8580.png,Normal/images/Normal-8580.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8580.png,Normal/masks/Normal-8580.png,True,False,299,299,L,118.41,bac8851be5f6b2acdfac8f3cdb8a9624
Normal-8581.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8581.png,Normal/images/Normal-8581.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8581.png,Normal/masks/Normal-8581.png,True,False,299,299,L,114.81,4eb22da254f631efacf13bd89d29ba57
Normal-8582.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8582.png,Normal/images/Normal-8582.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8582.png,Normal/masks/Normal-8582.png,True,False,299,299,L,107.55,30c07271f04ce3aa80456b632c4d7fa3
Normal-8583.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8583.png,Normal/images/Normal-8583.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8583.png,Normal/masks/Normal-8583.png,True,False,299,299,L,157.39,7f2cd8c471b349c0fa49a56594b34af3
Normal-8584.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8584.png,Normal/images/Normal-8584.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8584.png,Normal/masks/Normal-8584.png,True,False,299,299,L,123.52,7d40fd0b17210137f48c3bfb00a817af
Normal-8585.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8585.png,Normal/images/Normal-8585.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8585.png,Normal/masks/Normal-8585.png,True,False,299,299,L,109.87,ff5f5c5f5470ed65770456f9db617dcc
Normal-8586.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8586.png,Normal/images/Normal-8586.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8586.png,Normal/masks/Normal-8586.png,True,False,299,299,L,135.24,4368b47f4d045d3d58e02dc7c2238fc9
Normal-8587.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8587.png,Normal/images/Normal-8587.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8587.png,Normal/masks/Normal-8587.png,True,False,299,299,L,124.41,fad41ed0b1405534e968a83c2b768de8
Normal-8588.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8588.png,Normal/images/Normal-8588.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8588.png,Normal/masks/Normal-8588.png,True,False,299,299,L,128.3,8ede186621920bc8a14a36e6ada57d3d
Normal-8589.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8589.png,Normal/images/Normal-8589.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8589.png,Normal/masks/Normal-8589.png,True,False,299,299,L,151.81,fb412af503199901a645a3e8bbf74b77
Normal-859.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-859.png,Normal/images/Normal-859.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-859.png,Normal/masks/Normal-859.png,True,False,299,299,L,152.9,3082fa59eae7698d65cb32d3e957e023
Normal-8590.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8590.png,Normal/images/Normal-8590.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8590.png,Normal/masks/Normal-8590.png,True,False,299,299,L,120.8,4fcd7413e316b0774cc6507ace010c51
Normal-8591.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8591.png,Normal/images/Normal-8591.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8591.png,Normal/masks/Normal-8591.png,True,False,299,299,L,148.89,50dd7e91a25fc0def6e88a7cd34af264
Normal-8592.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8592.png,Normal/images/Normal-8592.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8592.png,Normal/masks/Normal-8592.png,True,False,299,299,L,164.27,765deabc64fe2c43d307dc24151022fd
Normal-8593.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8593.png,Normal/images/Normal-8593.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8593.png,Normal/masks/Normal-8593.png,True,False,299,299,L,113.28,63ae48bb258605b72df178de645ba86a
Normal-8594.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8594.png,Normal/images/Normal-8594.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8594.png,Normal/masks/Normal-8594.png,True,False,299,299,L,134.9,a1c9489f1b3ec42a33892cdee3fab99e
Normal-8595.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8595.png,Normal/images/Normal-8595.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8595.png,Normal/masks/Normal-8595.png,True,False,299,299,L,155.37,a9418ea10a81842d02bd33ddd086c5e9
Normal-8596.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8596.png,Normal/images/Normal-8596.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8596.png,Normal/masks/Normal-8596.png,True,False,299,299,L,127.68,f88016aab03334c848cb8494aad4f886
Normal-8597.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8597.png,Normal/images/Normal-8597.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8597.png,Normal/masks/Normal-8597.png,True,False,299,299,L,156.81,4ad8a0e4dc044c6aaec5e2c98c59a14d
Normal-8598.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8598.png,Normal/images/Normal-8598.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8598.png,Normal/masks/Normal-8598.png,True,False,299,299,L,110.78,542fa63a7ed94c2fee258848340c4a73
Normal-8599.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8599.png,Normal/images/Normal-8599.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8599.png,Normal/masks/Normal-8599.png,True,False,299,299,L,141.14,6d47efedc1ade14a2c812f37b9a98d79
Normal-86.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-86.png,Normal/images/Normal-86.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-86.png,Normal/masks/Normal-86.png,True,False,299,299,L,128.58,4b0378c21a5313d789d8890a663ae269
Normal-860.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-860.png,Normal/images/Normal-860.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-860.png,Normal/masks/Normal-860.png,True,False,299,299,L,140.81,b3b649dfb818f78a5ea7d64248d618a1
Normal-8600.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8600.png,Normal/images/Normal-8600.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8600.png,Normal/masks/Normal-8600.png,True,False,299,299,L,147.15,3df458350cea6ecb9c33f6eb9e98d0e3
Normal-8601.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8601.png,Normal/images/Normal-8601.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8601.png,Normal/masks/Normal-8601.png,True,False,299,299,L,121.88,89fa81bc896731c29cd2c1f5e15aff15
Normal-8602.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8602.png,Normal/images/Normal-8602.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8602.png,Normal/masks/Normal-8602.png,True,False,299,299,L,134.1,f3f88327941ca2760e4e208feb20499a
Normal-8603.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8603.png,Normal/images/Normal-8603.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8603.png,Normal/masks/Normal-8603.png,True,False,299,299,L,139.16,f3b69cd747b2ba4557fc701b951e7c65
Normal-8604.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8604.png,Normal/images/Normal-8604.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8604.png,Normal/masks/Normal-8604.png,True,False,299,299,L,135.03,d59ee702cad1e33aafdc9170af5331aa
Normal-8605.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8605.png,Normal/images/Normal-8605.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8605.png,Normal/masks/Normal-8605.png,True,False,299,299,L,138.12,43a63e49350a9d8394fe88eb41906b0a
Normal-8606.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8606.png,Normal/images/Normal-8606.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8606.png,Normal/masks/Normal-8606.png,True,False,299,299,L,166.38,8d821d556261af43fd7e03913a19bbd6
Normal-8607.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8607.png,Normal/images/Normal-8607.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8607.png,Normal/masks/Normal-8607.png,True,False,299,299,L,130.43,7229f04e2cdbc03f301b453b89197c50
Normal-8608.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8608.png,Normal/images/Normal-8608.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8608.png,Normal/masks/Normal-8608.png,True,False,299,299,L,137.88,50f6e7fe0702bf78b9bad0df12d9ba66
Normal-8609.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8609.png,Normal/images/Normal-8609.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8609.png,Normal/masks/Normal-8609.png,True,False,299,299,L,106.99,86acc80d4d23a96aad6bf60e86b93af9
Normal-861.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-861.png,Normal/images/Normal-861.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-861.png,Normal/masks/Normal-861.png,True,False,299,299,L,107.94,5fdc42c2332d426a80518d666ecec0a4
Normal-8610.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8610.png,Normal/images/Normal-8610.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8610.png,Normal/masks/Normal-8610.png,True,False,299,299,L,123.04,e611b340f3a0540ebc924bf38c7070de
Normal-8611.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8611.png,Normal/images/Normal-8611.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8611.png,Normal/masks/Normal-8611.png,True,False,299,299,L,97.14,72ad5ce439e6049611e5a3f612172f35
Normal-8612.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8612.png,Normal/images/Normal-8612.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8612.png,Normal/masks/Normal-8612.png,True,False,299,299,L,101.72,e42b7447effa490f4b3b577453da5ca8
Normal-8613.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8613.png,Normal/images/Normal-8613.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8613.png,Normal/masks/Normal-8613.png,True,False,299,299,L,143.64,ff7715fe6cd92193b67f1e68f6ceaeef
Normal-8614.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8614.png,Normal/images/Normal-8614.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8614.png,Normal/masks/Normal-8614.png,True,False,299,299,L,113.29,cc31cfc7477bfbb490be26a422ff2a5a
Normal-8615.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8615.png,Normal/images/Normal-8615.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8615.png,Normal/masks/Normal-8615.png,True,False,299,299,L,162.19,78eaf644266b1c1462a040ee29fc4ae3
Normal-8616.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8616.png,Normal/images/Normal-8616.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8616.png,Normal/masks/Normal-8616.png,True,False,299,299,L,122.37,a1d984ff73cb10dac99238e4984019ef
Normal-8617.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8617.png,Normal/images/Normal-8617.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8617.png,Normal/masks/Normal-8617.png,True,False,299,299,L,112.36,a1c6d19ef3c75518872b854273b7e472
Normal-8618.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8618.png,Normal/images/Normal-8618.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8618.png,Normal/masks/Normal-8618.png,True,False,299,299,L,145.87,5d24599000ce306ecd1a598d1e6868bb
Normal-8619.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8619.png,Normal/images/Normal-8619.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8619.png,Normal/masks/Normal-8619.png,True,False,299,299,L,121.23,1cd8e71b37d51a5d92174ebbc3286ee4
Normal-862.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-862.png,Normal/images/Normal-862.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-862.png,Normal/masks/Normal-862.png,True,False,299,299,L,130.14,fc0de14ad7eaa7d56abc9518b1380341
Normal-8620.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8620.png,Normal/images/Normal-8620.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8620.png,Normal/masks/Normal-8620.png,True,False,299,299,L,122.7,63f17088e69da1f9c686752908138c2a
Normal-8621.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8621.png,Normal/images/Normal-8621.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8621.png,Normal/masks/Normal-8621.png,True,False,299,299,L,167.9,739c5d1a9c5ed8065ab1494dd286acd6
Normal-8622.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8622.png,Normal/images/Normal-8622.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8622.png,Normal/masks/Normal-8622.png,True,False,299,299,L,99.8,2e007c9c247f7993431366552a00d423
Normal-8623.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8623.png,Normal/images/Normal-8623.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8623.png,Normal/masks/Normal-8623.png,True,False,299,299,L,161.56,506aa1021d3420f838d3e5414d954f19
Normal-8624.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8624.png,Normal/images/Normal-8624.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8624.png,Normal/masks/Normal-8624.png,True,False,299,299,L,102.94,0a04862b7da7d9676b96153d3aaa1c9f
Normal-8625.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8625.png,Normal/images/Normal-8625.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8625.png,Normal/masks/Normal-8625.png,True,False,299,299,L,110.69,625d493c875d5dff527ab0b9d5ff5b2d
Normal-8626.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8626.png,Normal/images/Normal-8626.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8626.png,Normal/masks/Normal-8626.png,True,False,299,299,L,134.7,eb35beaeba09f058a6b67f3c91e6e218
Normal-8627.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8627.png,Normal/images/Normal-8627.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8627.png,Normal/masks/Normal-8627.png,True,False,299,299,L,109.8,8e37a7d5edb3b4df014b155395e2fac8
Normal-8628.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8628.png,Normal/images/Normal-8628.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8628.png,Normal/masks/Normal-8628.png,True,False,299,299,L,123.4,cb7a5f8fecb6ad20e6c15e146f6b4bfa
Normal-8629.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8629.png,Normal/images/Normal-8629.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8629.png,Normal/masks/Normal-8629.png,True,False,299,299,L,114.05,40a0bffba6b2abea17b85dded9045f5c
Normal-863.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-863.png,Normal/images/Normal-863.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-863.png,Normal/masks/Normal-863.png,True,False,299,299,L,111.18,0dea583869b8d1c3009a6a092f78a734
Normal-8630.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8630.png,Normal/images/Normal-8630.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8630.png,Normal/masks/Normal-8630.png,True,False,299,299,L,146.83,4d6c4732dd7879755c2cef9194f484e9
Normal-8631.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8631.png,Normal/images/Normal-8631.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8631.png,Normal/masks/Normal-8631.png,True,False,299,299,L,152.03,666ad1d56b747996bb7863f012bdefaa
Normal-8632.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8632.png,Normal/images/Normal-8632.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8632.png,Normal/masks/Normal-8632.png,True,False,299,299,L,158.12,e521c821d8c68dce8347be4a845f4ad7
Normal-8633.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8633.png,Normal/images/Normal-8633.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8633.png,Normal/masks/Normal-8633.png,True,False,299,299,L,154.56,b1cdeed7a8f9f24aae269dc4653ea0c4
Normal-8634.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8634.png,Normal/images/Normal-8634.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8634.png,Normal/masks/Normal-8634.png,True,False,299,299,L,124.52,1aa5a505482a38fa95522d12cf67a329
Normal-8635.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8635.png,Normal/images/Normal-8635.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8635.png,Normal/masks/Normal-8635.png,True,False,299,299,L,109.71,610242fe796095b2b1f7a3f5cb087c83
Normal-8636.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8636.png,Normal/images/Normal-8636.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8636.png,Normal/masks/Normal-8636.png,True,False,299,299,L,158.82,e0902701b3073a66cbdbf12edd755594
Normal-8637.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8637.png,Normal/images/Normal-8637.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8637.png,Normal/masks/Normal-8637.png,True,False,299,299,L,125.82,be81457d4e1e13a7b65ed00e615ce5f4
Normal-8638.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8638.png,Normal/images/Normal-8638.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8638.png,Normal/masks/Normal-8638.png,True,False,299,299,L,113.94,0c3c969991ee9251d63efd62c333ef71
Normal-8639.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8639.png,Normal/images/Normal-8639.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8639.png,Normal/masks/Normal-8639.png,True,False,299,299,L,117.39,207b90c047d32ac6c21b5a8b895d68a8
Normal-864.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-864.png,Normal/images/Normal-864.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-864.png,Normal/masks/Normal-864.png,True,False,299,299,L,125.89,bcad8918f10acddf89496cbba07038b1
Normal-8640.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8640.png,Normal/images/Normal-8640.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8640.png,Normal/masks/Normal-8640.png,True,False,299,299,L,105.74,d88a935c58d6237039e47dbe6698d43e
Normal-8641.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8641.png,Normal/images/Normal-8641.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8641.png,Normal/masks/Normal-8641.png,True,False,299,299,L,147.4,5e335be59176ffe26fce85f445719bc3
Normal-8642.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8642.png,Normal/images/Normal-8642.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8642.png,Normal/masks/Normal-8642.png,True,False,299,299,L,96.69,33cc67c7230a08f9a6e492c79ed01d22
Normal-8643.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8643.png,Normal/images/Normal-8643.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8643.png,Normal/masks/Normal-8643.png,True,False,299,299,L,141.12,f9a8b209db58f52723f41dc3a953965b
Normal-8644.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8644.png,Normal/images/Normal-8644.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8644.png,Normal/masks/Normal-8644.png,True,False,299,299,L,164.1,07f532e84d3deeee1f51cb2f46633fe5
Normal-8645.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8645.png,Normal/images/Normal-8645.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8645.png,Normal/masks/Normal-8645.png,True,False,299,299,L,116.59,eebaa1144a3874ef5ee5246fb20e9331
Normal-8646.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8646.png,Normal/images/Normal-8646.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8646.png,Normal/masks/Normal-8646.png,True,False,299,299,L,141.18,f8503a4018521d5839a8bdae44ba22f2
Normal-8647.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8647.png,Normal/images/Normal-8647.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8647.png,Normal/masks/Normal-8647.png,True,False,299,299,L,132.42,1ce327c290b5b07ebdd22edbd5b82efc
Normal-8648.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8648.png,Normal/images/Normal-8648.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8648.png,Normal/masks/Normal-8648.png,True,False,299,299,L,117.15,25c3d9eb9941dff6a7d10f11db7b3dc1
Normal-8649.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8649.png,Normal/images/Normal-8649.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8649.png,Normal/masks/Normal-8649.png,True,False,299,299,L,123.18,c6af873d8345e873637700c7a7423bcb
Normal-865.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-865.png,Normal/images/Normal-865.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-865.png,Normal/masks/Normal-865.png,True,False,299,299,L,121.66,2518237e8b2dc4bee91e7c5fdf6d3def
Normal-8650.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8650.png,Normal/images/Normal-8650.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8650.png,Normal/masks/Normal-8650.png,True,False,299,299,L,120.71,de727eb939041069c6f8533a400379b7
Normal-8651.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8651.png,Normal/images/Normal-8651.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8651.png,Normal/masks/Normal-8651.png,True,False,299,299,L,122.85,e27db0391e62f178a8ebca26b92536bc
Normal-8652.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8652.png,Normal/images/Normal-8652.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8652.png,Normal/masks/Normal-8652.png,True,False,299,299,L,150.28,71645e2963b1ef3215cd9b3540eb2660
Normal-8653.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8653.png,Normal/images/Normal-8653.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8653.png,Normal/masks/Normal-8653.png,True,False,299,299,L,120.65,7bb8ea460e3390bc590f723534bd8c49
Normal-8654.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8654.png,Normal/images/Normal-8654.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8654.png,Normal/masks/Normal-8654.png,True,False,299,299,L,166.21,f6da1a541804bf623058290ed13e6353
Normal-8655.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8655.png,Normal/images/Normal-8655.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8655.png,Normal/masks/Normal-8655.png,True,False,299,299,L,108.08,ecfbeb3e3ea5de77c7ce485497f213fd
Normal-8656.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8656.png,Normal/images/Normal-8656.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8656.png,Normal/masks/Normal-8656.png,True,False,299,299,L,119.4,1fc3091571ff698bf7abfb86d021af61
Normal-8657.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8657.png,Normal/images/Normal-8657.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8657.png,Normal/masks/Normal-8657.png,True,False,299,299,L,140.35,66cd6a16717a22af5f508001217dc528
Normal-8658.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8658.png,Normal/images/Normal-8658.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8658.png,Normal/masks/Normal-8658.png,True,False,299,299,L,155.19,d497aeef40cd22ee9961f2bd5ffb2bc3
Normal-8659.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8659.png,Normal/images/Normal-8659.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8659.png,Normal/masks/Normal-8659.png,True,False,299,299,L,109.49,c90d13efbd2e8d8d64c3c3571b251395
Normal-866.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-866.png,Normal/images/Normal-866.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-866.png,Normal/masks/Normal-866.png,True,False,299,299,L,137.58,a4cba479eb2c431f0bd447db49eede7a
Normal-8660.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8660.png,Normal/images/Normal-8660.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8660.png,Normal/masks/Normal-8660.png,True,False,299,299,L,113.51,c59aacc2570a7357dcb9e91ee10e5ce6
Normal-8661.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8661.png,Normal/images/Normal-8661.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8661.png,Normal/masks/Normal-8661.png,True,False,299,299,L,168.41,50c020f92f134001fb8988e000126293
Normal-8662.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8662.png,Normal/images/Normal-8662.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8662.png,Normal/masks/Normal-8662.png,True,False,299,299,L,109.35,5dd52ea7e2dcb5c2973e1328421498ae
Normal-8663.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8663.png,Normal/images/Normal-8663.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8663.png,Normal/masks/Normal-8663.png,True,False,299,299,L,99.42,cd070dd6e0a8f429eb496ca32b50b422
Normal-8664.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8664.png,Normal/images/Normal-8664.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8664.png,Normal/masks/Normal-8664.png,True,False,299,299,L,112.38,7e935b8252b0a26730ba2f35099bc38e
Normal-8665.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8665.png,Normal/images/Normal-8665.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8665.png,Normal/masks/Normal-8665.png,True,False,299,299,L,101.5,ff9075181e8129cec4aa0b010fc04977
Normal-8666.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8666.png,Normal/images/Normal-8666.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8666.png,Normal/masks/Normal-8666.png,True,False,299,299,L,152.92,eefd034e9fef22bce9b22111c2a8fca4
Normal-8667.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8667.png,Normal/images/Normal-8667.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8667.png,Normal/masks/Normal-8667.png,True,False,299,299,L,167.68,e5b5a045cd58a7425e9735d2a91d08d2
Normal-8668.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8668.png,Normal/images/Normal-8668.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8668.png,Normal/masks/Normal-8668.png,True,False,299,299,L,145.07,9216e875043952fa81d6a91a17f66c6a
Normal-8669.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8669.png,Normal/images/Normal-8669.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8669.png,Normal/masks/Normal-8669.png,True,False,299,299,L,128.86,7132996a340d7d67c60ccba717fdfa0a
Normal-867.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-867.png,Normal/images/Normal-867.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-867.png,Normal/masks/Normal-867.png,True,False,299,299,L,122.73,e7f441f87fc8f5c2c9ae99ac9183a9c8
Normal-8670.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8670.png,Normal/images/Normal-8670.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8670.png,Normal/masks/Normal-8670.png,True,False,299,299,L,150.86,268aeaaa436ff20574be70add60f9785
Normal-8671.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8671.png,Normal/images/Normal-8671.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8671.png,Normal/masks/Normal-8671.png,True,False,299,299,L,143.95,d5b86444bc5c22c29bb9a6b9690b9fa5
Normal-8672.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8672.png,Normal/images/Normal-8672.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8672.png,Normal/masks/Normal-8672.png,True,False,299,299,L,167.25,938d293da5426e11c3f02be5afa049ed
Normal-8673.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8673.png,Normal/images/Normal-8673.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8673.png,Normal/masks/Normal-8673.png,True,False,299,299,L,170.91,15400c098178f35730a49e4288809f2d
Normal-8674.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8674.png,Normal/images/Normal-8674.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8674.png,Normal/masks/Normal-8674.png,True,False,299,299,L,124.1,b9387df5acc26ac8289600f78ceaa8ed
Normal-8675.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8675.png,Normal/images/Normal-8675.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8675.png,Normal/masks/Normal-8675.png,True,False,299,299,L,118.78,9faaf14d8a56bbbc3b2a26a609128411
Normal-8676.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8676.png,Normal/images/Normal-8676.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8676.png,Normal/masks/Normal-8676.png,True,False,299,299,L,134.52,4c32b29e8e790fdf511b1634f5ac6fad
Normal-8677.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8677.png,Normal/images/Normal-8677.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8677.png,Normal/masks/Normal-8677.png,True,False,299,299,L,180.1,2de79351ce2360e94cdce07ff036c6ce
Normal-8678.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8678.png,Normal/images/Normal-8678.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8678.png,Normal/masks/Normal-8678.png,True,False,299,299,L,141.32,ac5b4b19fc084b273a1e4cc291cdd0ac
Normal-8679.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8679.png,Normal/images/Normal-8679.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8679.png,Normal/masks/Normal-8679.png,True,False,299,299,L,109.85,90f3e7f45c179ba678bbb0b0ec727bc0
Normal-868.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-868.png,Normal/images/Normal-868.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-868.png,Normal/masks/Normal-868.png,True,False,299,299,L,124.62,55da08b5574174651c1983bc5437845f
Normal-8680.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8680.png,Normal/images/Normal-8680.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8680.png,Normal/masks/Normal-8680.png,True,False,299,299,L,111.37,bfc578a18e90bb1afd70da06c0cc7174
Normal-8681.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8681.png,Normal/images/Normal-8681.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8681.png,Normal/masks/Normal-8681.png,True,False,299,299,L,116.77,4c61f0677290a48a15ab194234041ef3
Normal-8682.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8682.png,Normal/images/Normal-8682.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8682.png,Normal/masks/Normal-8682.png,True,False,299,299,L,141.81,b49787d3bb198db6ca0362d0a90126d2
Normal-8683.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8683.png,Normal/images/Normal-8683.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8683.png,Normal/masks/Normal-8683.png,True,False,299,299,L,125.6,eccd7c09e288b1ef103f766e778f2d52
Normal-8684.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8684.png,Normal/images/Normal-8684.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8684.png,Normal/masks/Normal-8684.png,True,False,299,299,L,142.34,683ee66f6b9b1b34c4a24d60f6da4729
Normal-8685.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8685.png,Normal/images/Normal-8685.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8685.png,Normal/masks/Normal-8685.png,True,False,299,299,L,102.84,da8fe1589a1c161e9e7246dccc78952b
Normal-8686.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8686.png,Normal/images/Normal-8686.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8686.png,Normal/masks/Normal-8686.png,True,False,299,299,L,129.23,c9b774c071175f96f7a7804ee70a154f
Normal-8687.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8687.png,Normal/images/Normal-8687.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8687.png,Normal/masks/Normal-8687.png,True,False,299,299,L,150.4,ec0ddd286be3b101208cd8e55d598301
Normal-8688.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8688.png,Normal/images/Normal-8688.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8688.png,Normal/masks/Normal-8688.png,True,False,299,299,L,129.57,323128f3c99a93a7c867bdc1f5e809ed
Normal-8689.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8689.png,Normal/images/Normal-8689.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8689.png,Normal/masks/Normal-8689.png,True,False,299,299,L,153.41,5273fc5198366ec9cb49b1ee517e8fcd
Normal-869.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-869.png,Normal/images/Normal-869.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-869.png,Normal/masks/Normal-869.png,True,False,299,299,L,109.05,ed1bd254524354ad300533ba6074891d
Normal-8690.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8690.png,Normal/images/Normal-8690.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8690.png,Normal/masks/Normal-8690.png,True,False,299,299,L,119.95,2d699f5995e92ab04a3c113bb94227ef
Normal-8691.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8691.png,Normal/images/Normal-8691.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8691.png,Normal/masks/Normal-8691.png,True,False,299,299,L,141.19,b039173dbc58c8f05442af97a56205d1
Normal-8692.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8692.png,Normal/images/Normal-8692.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8692.png,Normal/masks/Normal-8692.png,True,False,299,299,L,110.75,9e3150710c56315c3ce1e119ece232af
Normal-8693.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8693.png,Normal/images/Normal-8693.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8693.png,Normal/masks/Normal-8693.png,True,False,299,299,L,101.66,cca79aba678449ca131e506ac126d186
Normal-8694.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8694.png,Normal/images/Normal-8694.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8694.png,Normal/masks/Normal-8694.png,True,False,299,299,L,103.47,6152785ae66018c55c720b992d22e72f
Normal-8695.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8695.png,Normal/images/Normal-8695.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8695.png,Normal/masks/Normal-8695.png,True,False,299,299,L,169.92,640f67edef46113260b744d4885abfcc
Normal-8696.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8696.png,Normal/images/Normal-8696.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8696.png,Normal/masks/Normal-8696.png,True,False,299,299,L,103.62,6697f2f311321ad7df01d04504d03e8f
Normal-8697.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8697.png,Normal/images/Normal-8697.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8697.png,Normal/masks/Normal-8697.png,True,False,299,299,L,104.13,f534a399521f252f583b172c4decd0b4
Normal-8698.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8698.png,Normal/images/Normal-8698.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8698.png,Normal/masks/Normal-8698.png,True,False,299,299,L,137.14,b5da681ca6cea687592e93664b10de45
Normal-8699.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8699.png,Normal/images/Normal-8699.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8699.png,Normal/masks/Normal-8699.png,True,False,299,299,L,100.36,4a1a2589e662fa92361f86d5ad45067d
Normal-87.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-87.png,Normal/images/Normal-87.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-87.png,Normal/masks/Normal-87.png,True,False,299,299,L,116.61,65371a377f167d20fb04113c6cc970b0
Normal-870.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-870.png,Normal/images/Normal-870.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-870.png,Normal/masks/Normal-870.png,True,False,299,299,L,134.62,f7a3ea74e9ceacf588ec3f2ee3f13588
Normal-8700.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8700.png,Normal/images/Normal-8700.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8700.png,Normal/masks/Normal-8700.png,True,False,299,299,L,99.26,ffd49bd656ca9273e973ec916ca975b2
Normal-8701.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8701.png,Normal/images/Normal-8701.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8701.png,Normal/masks/Normal-8701.png,True,False,299,299,L,160.3,d462d9096765641c3bdd6937bde8da09
Normal-8702.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8702.png,Normal/images/Normal-8702.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8702.png,Normal/masks/Normal-8702.png,True,False,299,299,L,111.4,c6f472113e0f72b61f44e5c955970a9b
Normal-8703.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8703.png,Normal/images/Normal-8703.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8703.png,Normal/masks/Normal-8703.png,True,False,299,299,L,105.59,07ee27ceea345bcc9ceb7cc7c9dde8a2
Normal-8704.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8704.png,Normal/images/Normal-8704.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8704.png,Normal/masks/Normal-8704.png,True,False,299,299,L,152.82,b5f68855fada8640ac12f0f037eeac65
Normal-8705.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8705.png,Normal/images/Normal-8705.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8705.png,Normal/masks/Normal-8705.png,True,False,299,299,L,136.06,119c52c2c144d9378b08f2ad71327c1d
Normal-8706.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8706.png,Normal/images/Normal-8706.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8706.png,Normal/masks/Normal-8706.png,True,False,299,299,L,134.27,17c73ea4cb0d595e8d579c9d41ab309d
Normal-8707.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8707.png,Normal/images/Normal-8707.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8707.png,Normal/masks/Normal-8707.png,True,False,299,299,L,154.23,22681609c5b680d3c31f625c491e2128
Normal-8708.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8708.png,Normal/images/Normal-8708.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8708.png,Normal/masks/Normal-8708.png,True,False,299,299,L,105.02,eb857707ac642e5ffce82e167fabb622
Normal-8709.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8709.png,Normal/images/Normal-8709.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8709.png,Normal/masks/Normal-8709.png,True,False,299,299,L,172.04,ebabba629140b12f7415989456899fcd
Normal-871.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-871.png,Normal/images/Normal-871.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-871.png,Normal/masks/Normal-871.png,True,False,299,299,L,145.57,4eb7b95cc9340d8f586c41da36b2b23f
Normal-8710.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8710.png,Normal/images/Normal-8710.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8710.png,Normal/masks/Normal-8710.png,True,False,299,299,L,169.06,5a18bdd07e7809b58888926e9267dcee
Normal-8711.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8711.png,Normal/images/Normal-8711.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8711.png,Normal/masks/Normal-8711.png,True,False,299,299,L,133.56,b47fa319d25db01dd16cb4ed29460f2f
Normal-8712.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8712.png,Normal/images/Normal-8712.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8712.png,Normal/masks/Normal-8712.png,True,False,299,299,L,152.57,7cc53c0e34d152a30e8760a2ad27b9db
Normal-8713.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8713.png,Normal/images/Normal-8713.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8713.png,Normal/masks/Normal-8713.png,True,False,299,299,L,153.98,fdd777efeb4f02a4e3b3381e3efb792c
Normal-8714.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8714.png,Normal/images/Normal-8714.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8714.png,Normal/masks/Normal-8714.png,True,False,299,299,L,172.43,1d895204c05d658e47cc997f60495123
Normal-8715.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8715.png,Normal/images/Normal-8715.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8715.png,Normal/masks/Normal-8715.png,True,False,299,299,L,118.18,53a1dce018b363443df02918f146249e
Normal-8716.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8716.png,Normal/images/Normal-8716.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8716.png,Normal/masks/Normal-8716.png,True,False,299,299,L,165.95,bdb35b2b763c9169a75698b6b2b913bf
Normal-8717.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8717.png,Normal/images/Normal-8717.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8717.png,Normal/masks/Normal-8717.png,True,False,299,299,L,110.01,3406e89a9d9441fa9af46e07c3b3c33f
Normal-8718.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8718.png,Normal/images/Normal-8718.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8718.png,Normal/masks/Normal-8718.png,True,False,299,299,L,163.17,6be2468f515f94916c66d510a1092062
Normal-8719.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8719.png,Normal/images/Normal-8719.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8719.png,Normal/masks/Normal-8719.png,True,False,299,299,L,138.58,f0a38ee9cb33cb9af5f8b2b8e1b93e12
Normal-872.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-872.png,Normal/images/Normal-872.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-872.png,Normal/masks/Normal-872.png,True,False,299,299,L,99.63,f6372f7235310dc58775c64d6f692548
Normal-8720.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8720.png,Normal/images/Normal-8720.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8720.png,Normal/masks/Normal-8720.png,True,False,299,299,L,165.4,2bb3399268369879d1b58f694da00cdb
Normal-8721.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8721.png,Normal/images/Normal-8721.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8721.png,Normal/masks/Normal-8721.png,True,False,299,299,L,153.93,01c5e66a60567deabe4f5eb07a162130
Normal-8722.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8722.png,Normal/images/Normal-8722.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8722.png,Normal/masks/Normal-8722.png,True,False,299,299,L,155.21,1657276cdcf42942cdb158bd6b7eda31
Normal-8723.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8723.png,Normal/images/Normal-8723.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8723.png,Normal/masks/Normal-8723.png,True,False,299,299,L,131.81,be7f5c0a100b40b79bd3a77b93aa7e14
Normal-8724.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8724.png,Normal/images/Normal-8724.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8724.png,Normal/masks/Normal-8724.png,True,False,299,299,L,115.99,fb2ae8bb84616123bddf21c669fcf14c
Normal-8725.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8725.png,Normal/images/Normal-8725.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8725.png,Normal/masks/Normal-8725.png,True,False,299,299,L,168.04,cd44a9c606f5dda9cc181e4056a454c0
Normal-8726.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8726.png,Normal/images/Normal-8726.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8726.png,Normal/masks/Normal-8726.png,True,False,299,299,L,161.95,dbfaed5dbcae12fc12b3b8272bb85d99
Normal-8727.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8727.png,Normal/images/Normal-8727.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8727.png,Normal/masks/Normal-8727.png,True,False,299,299,L,117.44,1875557f8dc9741413813631cffbf24f
Normal-8728.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8728.png,Normal/images/Normal-8728.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8728.png,Normal/masks/Normal-8728.png,True,False,299,299,L,170.58,d477641893a2f8b6ce38db93228b1eaf
Normal-8729.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8729.png,Normal/images/Normal-8729.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8729.png,Normal/masks/Normal-8729.png,True,False,299,299,L,141.23,dac5182e61bf8109fec91bf850e47e08
Normal-873.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-873.png,Normal/images/Normal-873.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-873.png,Normal/masks/Normal-873.png,True,False,299,299,L,147.19,cc4f6a133a03c3c7241278172781fb29
Normal-8730.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8730.png,Normal/images/Normal-8730.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8730.png,Normal/masks/Normal-8730.png,True,False,299,299,L,131.15,3c8a40c99e486c7549e247b9fe5f5926
Normal-8731.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8731.png,Normal/images/Normal-8731.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8731.png,Normal/masks/Normal-8731.png,True,False,299,299,L,104.28,6afb61b59ded50e1dd3b3f1a77ffa3c6
Normal-8732.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8732.png,Normal/images/Normal-8732.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8732.png,Normal/masks/Normal-8732.png,True,False,299,299,L,126.65,ee733cdeac75494dcfc677c040edba07
Normal-8733.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8733.png,Normal/images/Normal-8733.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8733.png,Normal/masks/Normal-8733.png,True,False,299,299,L,133.06,ce02a7a5699977eb801a297344086cb8
Normal-8734.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8734.png,Normal/images/Normal-8734.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8734.png,Normal/masks/Normal-8734.png,True,False,299,299,L,123.28,0a4a151e260afe6bd8480fdfd727a6b5
Normal-8735.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8735.png,Normal/images/Normal-8735.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8735.png,Normal/masks/Normal-8735.png,True,False,299,299,L,166.43,8363a3355f93a878fd0eb5aaa23f8ecb
Normal-8736.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8736.png,Normal/images/Normal-8736.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8736.png,Normal/masks/Normal-8736.png,True,False,299,299,L,108.87,50968a575fb534164d3d8035093c3792
Normal-8737.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8737.png,Normal/images/Normal-8737.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8737.png,Normal/masks/Normal-8737.png,True,False,299,299,L,169.41,446c2045172624181e9b73032da89d07
Normal-8738.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8738.png,Normal/images/Normal-8738.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8738.png,Normal/masks/Normal-8738.png,True,False,299,299,L,99.59,a70cf1fbc41c6ecd54bc158ba874cd1f
Normal-8739.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8739.png,Normal/images/Normal-8739.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8739.png,Normal/masks/Normal-8739.png,True,False,299,299,L,113.06,b8c7f333d3ed2b7e8b03e7323cc4dab3
Normal-874.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-874.png,Normal/images/Normal-874.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-874.png,Normal/masks/Normal-874.png,True,False,299,299,L,103.04,d96b73c6efbfd6014b89c37077097d8e
Normal-8740.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8740.png,Normal/images/Normal-8740.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8740.png,Normal/masks/Normal-8740.png,True,False,299,299,L,166.24,bc3d9666ded76d93eb7ce0a6e5aa11b1
Normal-8741.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8741.png,Normal/images/Normal-8741.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8741.png,Normal/masks/Normal-8741.png,True,False,299,299,L,134.52,351c6da162218a158269486024b64da0
Normal-8742.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8742.png,Normal/images/Normal-8742.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8742.png,Normal/masks/Normal-8742.png,True,False,299,299,L,122.4,ab944faecb66a4f35723abf65c4f2642
Normal-8743.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8743.png,Normal/images/Normal-8743.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8743.png,Normal/masks/Normal-8743.png,True,False,299,299,L,125.96,2bb90db72a9c48232fd7fa9bf5d8668b
Normal-8744.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8744.png,Normal/images/Normal-8744.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8744.png,Normal/masks/Normal-8744.png,True,False,299,299,L,153.5,daf306bd3d9ccb6dc9e34d69279ffd8c
Normal-8745.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8745.png,Normal/images/Normal-8745.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8745.png,Normal/masks/Normal-8745.png,True,False,299,299,L,114.36,15e3d60b8b2a4f1c165977d41331c690
Normal-8746.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8746.png,Normal/images/Normal-8746.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8746.png,Normal/masks/Normal-8746.png,True,False,299,299,L,164.83,0812e23e073a7e4d9f902bbe731cf4b8
Normal-8747.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8747.png,Normal/images/Normal-8747.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8747.png,Normal/masks/Normal-8747.png,True,False,299,299,L,113.14,c06a841fa39d1469d7259a37009cc8ff
Normal-8748.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8748.png,Normal/images/Normal-8748.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8748.png,Normal/masks/Normal-8748.png,True,False,299,299,L,99.98,1780a97fdcb2694092d407be5f433f42
Normal-8749.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8749.png,Normal/images/Normal-8749.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8749.png,Normal/masks/Normal-8749.png,True,False,299,299,L,132.14,e048a05183fe8359818651732c809750
Normal-875.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-875.png,Normal/images/Normal-875.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-875.png,Normal/masks/Normal-875.png,True,False,299,299,L,132.72,9f928c87562372245ac0ebab03edda7c
Normal-8750.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8750.png,Normal/images/Normal-8750.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8750.png,Normal/masks/Normal-8750.png,True,False,299,299,L,154.2,d874cd847d36254bf7daa8a0ac9044fb
Normal-8751.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8751.png,Normal/images/Normal-8751.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8751.png,Normal/masks/Normal-8751.png,True,False,299,299,L,150.06,9638d61491e02b9eb3d2d6de7aa2f700
Normal-8752.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8752.png,Normal/images/Normal-8752.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8752.png,Normal/masks/Normal-8752.png,True,False,299,299,L,134.83,faf5e2f467ae8e4603259c487ff97d98
Normal-8753.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8753.png,Normal/images/Normal-8753.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8753.png,Normal/masks/Normal-8753.png,True,False,299,299,L,120.07,99fc5f5dafe8b5db1f8925d11334e26d
Normal-8754.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8754.png,Normal/images/Normal-8754.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8754.png,Normal/masks/Normal-8754.png,True,False,299,299,L,163.55,13ee441ee0c34183a00b9af50db65b93
Normal-8755.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8755.png,Normal/images/Normal-8755.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8755.png,Normal/masks/Normal-8755.png,True,False,299,299,L,137.15,98479e1c0a0b72af5f6435e9691d1b38
Normal-8756.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8756.png,Normal/images/Normal-8756.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8756.png,Normal/masks/Normal-8756.png,True,False,299,299,L,144.02,73d880ed9a99d097b4f72b2419f014e3
Normal-8757.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8757.png,Normal/images/Normal-8757.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8757.png,Normal/masks/Normal-8757.png,True,False,299,299,L,134.54,247b6733c9658d9d6cb6a262ff4ad433
Normal-8758.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8758.png,Normal/images/Normal-8758.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8758.png,Normal/masks/Normal-8758.png,True,False,299,299,L,118.0,f55f7d0f42c012d98e4d92a998b45e6c
Normal-8759.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8759.png,Normal/images/Normal-8759.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8759.png,Normal/masks/Normal-8759.png,True,False,299,299,L,159.7,9500adcb7ca3a641807fdc1cb97d9819
Normal-876.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-876.png,Normal/images/Normal-876.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-876.png,Normal/masks/Normal-876.png,True,False,299,299,L,123.76,ad667c4a50103cfc9976baadb2352d54
Normal-8760.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8760.png,Normal/images/Normal-8760.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8760.png,Normal/masks/Normal-8760.png,True,False,299,299,L,116.63,d93109ab133f07675f3b73ce58efeeb0
Normal-8761.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8761.png,Normal/images/Normal-8761.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8761.png,Normal/masks/Normal-8761.png,True,False,299,299,L,180.25,7cfb93e4e00786c8d6fb9967a83dd5f2
Normal-8762.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8762.png,Normal/images/Normal-8762.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8762.png,Normal/masks/Normal-8762.png,True,False,299,299,L,159.71,19475df9bc495924a7ffe0b67f07c2ae
Normal-8763.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8763.png,Normal/images/Normal-8763.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8763.png,Normal/masks/Normal-8763.png,True,False,299,299,L,166.19,2b30d734ffc793748235d413b1214070
Normal-8764.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8764.png,Normal/images/Normal-8764.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8764.png,Normal/masks/Normal-8764.png,True,False,299,299,L,123.3,ff9cd98fd4d31cc143c8fe6e0e096315
Normal-8765.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8765.png,Normal/images/Normal-8765.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8765.png,Normal/masks/Normal-8765.png,True,False,299,299,L,115.58,9c7e081197a132b4399bc675b1a6dc51
Normal-8766.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8766.png,Normal/images/Normal-8766.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8766.png,Normal/masks/Normal-8766.png,True,False,299,299,L,99.35,5cb445c465a03e419ad091d206053c8f
Normal-8767.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8767.png,Normal/images/Normal-8767.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8767.png,Normal/masks/Normal-8767.png,True,False,299,299,L,126.21,75604f608a94869e0c4feee13d066e3b
Normal-8768.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8768.png,Normal/images/Normal-8768.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8768.png,Normal/masks/Normal-8768.png,True,False,299,299,L,119.11,406eb23e18efa27ec70cd075691e967d
Normal-8769.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8769.png,Normal/images/Normal-8769.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8769.png,Normal/masks/Normal-8769.png,True,False,299,299,L,180.27,ebc6d4da960cb27ba077b4cc3805144a
Normal-877.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-877.png,Normal/images/Normal-877.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-877.png,Normal/masks/Normal-877.png,True,False,299,299,L,105.96,678b5aa445da869f5af25cb077d613f5
Normal-8770.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8770.png,Normal/images/Normal-8770.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8770.png,Normal/masks/Normal-8770.png,True,False,299,299,L,170.13,7503bea818013cec4b19ed84320d7256
Normal-8771.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8771.png,Normal/images/Normal-8771.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8771.png,Normal/masks/Normal-8771.png,True,False,299,299,L,99.93,a82c56ad2a553f9bad4894d7e093fd33
Normal-8772.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8772.png,Normal/images/Normal-8772.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8772.png,Normal/masks/Normal-8772.png,True,False,299,299,L,134.85,0533a93c914cd05617172801cbd6fd5e
Normal-8773.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8773.png,Normal/images/Normal-8773.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8773.png,Normal/masks/Normal-8773.png,True,False,299,299,L,119.04,3a3a8dc9a8c7fe54d9a779f6c57b6dac
Normal-8774.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8774.png,Normal/images/Normal-8774.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8774.png,Normal/masks/Normal-8774.png,True,False,299,299,L,122.67,c9251f4ce30a83c89a001bde21dc1871
Normal-8775.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8775.png,Normal/images/Normal-8775.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8775.png,Normal/masks/Normal-8775.png,True,False,299,299,L,116.3,94781db320fe44006456554d67855cde
Normal-8776.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8776.png,Normal/images/Normal-8776.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8776.png,Normal/masks/Normal-8776.png,True,False,299,299,L,140.13,d2f95ad3f12faa773f8e723ae9693c60
Normal-8777.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8777.png,Normal/images/Normal-8777.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8777.png,Normal/masks/Normal-8777.png,True,False,299,299,L,140.27,11e637623048bebe4892f7bdeb7ca83d
Normal-8778.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8778.png,Normal/images/Normal-8778.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8778.png,Normal/masks/Normal-8778.png,True,False,299,299,L,131.78,286666ba6b8592db6d4b171c3aa2b5b7
Normal-8779.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8779.png,Normal/images/Normal-8779.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8779.png,Normal/masks/Normal-8779.png,True,False,299,299,L,146.21,ff2759518db535b0ae37fdac0410fd07
Normal-878.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-878.png,Normal/images/Normal-878.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-878.png,Normal/masks/Normal-878.png,True,False,299,299,L,129.14,4d73aa8847aafb8186bab595ff897d97
Normal-8780.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8780.png,Normal/images/Normal-8780.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8780.png,Normal/masks/Normal-8780.png,True,False,299,299,L,137.09,4a03614473d112831fa5f97dad186f60
Normal-8781.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8781.png,Normal/images/Normal-8781.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8781.png,Normal/masks/Normal-8781.png,True,False,299,299,L,120.56,85307b8e4cd422da8b5bc8f85854ab9d
Normal-8782.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8782.png,Normal/images/Normal-8782.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8782.png,Normal/masks/Normal-8782.png,True,False,299,299,L,157.8,4d3e622c5b264005dbae18197b98cf7f
Normal-8783.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8783.png,Normal/images/Normal-8783.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8783.png,Normal/masks/Normal-8783.png,True,False,299,299,L,164.95,48655e64d5f08a6fa448657050e7fdc4
Normal-8784.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8784.png,Normal/images/Normal-8784.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8784.png,Normal/masks/Normal-8784.png,True,False,299,299,L,177.19,2086e5cb46033d0aa0cee8564a59b2b1
Normal-8785.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8785.png,Normal/images/Normal-8785.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8785.png,Normal/masks/Normal-8785.png,True,False,299,299,L,117.98,a32daee2d0a5bd4a1aff69d89a6559c4
Normal-8786.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8786.png,Normal/images/Normal-8786.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8786.png,Normal/masks/Normal-8786.png,True,False,299,299,L,149.93,f88053564839f6f0b6374ff158ba80ad
Normal-8787.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8787.png,Normal/images/Normal-8787.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8787.png,Normal/masks/Normal-8787.png,True,False,299,299,L,175.79,2cc5c689e0e3613b2303b8564c3d3a3f
Normal-8788.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8788.png,Normal/images/Normal-8788.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8788.png,Normal/masks/Normal-8788.png,True,False,299,299,L,90.15,c2407af155faa5c6221256875f756170
Normal-8789.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8789.png,Normal/images/Normal-8789.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8789.png,Normal/masks/Normal-8789.png,True,False,299,299,L,92.55,ddb0a14a8ff9dacd5497bc289aecbb39
Normal-879.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-879.png,Normal/images/Normal-879.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-879.png,Normal/masks/Normal-879.png,True,False,299,299,L,129.91,e1a7543b647c40fb055dd020ccfc4145
Normal-8790.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8790.png,Normal/images/Normal-8790.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8790.png,Normal/masks/Normal-8790.png,True,False,299,299,L,112.57,14c994160f75c21d2f074b67ba8edeea
Normal-8791.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8791.png,Normal/images/Normal-8791.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8791.png,Normal/masks/Normal-8791.png,True,False,299,299,L,138.29,4a05b92c1b9e81de253bf2ac81bc250c
Normal-8792.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8792.png,Normal/images/Normal-8792.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8792.png,Normal/masks/Normal-8792.png,True,False,299,299,L,94.61,94817d170b195f03c0ddea45f8a16b0d
Normal-8793.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8793.png,Normal/images/Normal-8793.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8793.png,Normal/masks/Normal-8793.png,True,False,299,299,L,139.5,bc109e96c34f30271ffc7e8c9207328d
Normal-8794.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8794.png,Normal/images/Normal-8794.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8794.png,Normal/masks/Normal-8794.png,True,False,299,299,L,133.34,babe879d9e972d1b48b9a0db936c8e97
Normal-8795.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8795.png,Normal/images/Normal-8795.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8795.png,Normal/masks/Normal-8795.png,True,False,299,299,L,118.31,769dfea240fbb355b2aa67322c76b2a9
Normal-8796.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8796.png,Normal/images/Normal-8796.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8796.png,Normal/masks/Normal-8796.png,True,False,299,299,L,116.4,2af8bf125b0283c1a947d78ddf7aa37c
Normal-8797.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8797.png,Normal/images/Normal-8797.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8797.png,Normal/masks/Normal-8797.png,True,False,299,299,L,123.97,afcbb80f035b523f9ff3dc2ae27c5d8d
Normal-8798.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8798.png,Normal/images/Normal-8798.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8798.png,Normal/masks/Normal-8798.png,True,False,299,299,L,123.19,41cf438525ee3cf0e8d10330aec51974
Normal-8799.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8799.png,Normal/images/Normal-8799.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8799.png,Normal/masks/Normal-8799.png,True,False,299,299,L,133.36,b9b94519e168d3f4f22bbef4f75aa383
Normal-88.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-88.png,Normal/images/Normal-88.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-88.png,Normal/masks/Normal-88.png,True,False,299,299,L,132.57,87260a894f906da6f9d0167af826ce68
Normal-880.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-880.png,Normal/images/Normal-880.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-880.png,Normal/masks/Normal-880.png,True,False,299,299,L,107.26,ad42b9c2f860e3a7b8da8b5140c2609d
Normal-8800.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8800.png,Normal/images/Normal-8800.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8800.png,Normal/masks/Normal-8800.png,True,False,299,299,L,132.64,276c97fbc325c6325c98a39e9f215250
Normal-8801.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8801.png,Normal/images/Normal-8801.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8801.png,Normal/masks/Normal-8801.png,True,False,299,299,L,141.44,cdfd941234460ea3e7b901bd00dd25bf
Normal-8802.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8802.png,Normal/images/Normal-8802.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8802.png,Normal/masks/Normal-8802.png,True,False,299,299,L,117.78,fb6dfae4b05e9268334fd3705637a89d
Normal-8803.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8803.png,Normal/images/Normal-8803.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8803.png,Normal/masks/Normal-8803.png,True,False,299,299,L,127.82,79fdef852ed19d0b94ee1d3d18a9d5f2
Normal-8804.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8804.png,Normal/images/Normal-8804.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8804.png,Normal/masks/Normal-8804.png,True,False,299,299,L,138.59,e5afba6ce6acd7c2b5900af313fd2bef
Normal-8805.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8805.png,Normal/images/Normal-8805.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8805.png,Normal/masks/Normal-8805.png,True,False,299,299,L,104.92,5f52c2c9820e466df537b54608eb7505
Normal-8806.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8806.png,Normal/images/Normal-8806.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8806.png,Normal/masks/Normal-8806.png,True,False,299,299,L,132.73,e0ee2b23fbeff3fb7276e703e12e7b2d
Normal-8807.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8807.png,Normal/images/Normal-8807.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8807.png,Normal/masks/Normal-8807.png,True,False,299,299,L,124.13,5da93acca528a3ee0b302383b9b49af3
Normal-8808.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8808.png,Normal/images/Normal-8808.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8808.png,Normal/masks/Normal-8808.png,True,False,299,299,L,134.38,bee122d6bde56121902101d10d2325b2
Normal-8809.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8809.png,Normal/images/Normal-8809.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8809.png,Normal/masks/Normal-8809.png,True,False,299,299,L,119.5,ba101b610096ac1b8af3b4a53c289b2b
Normal-881.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-881.png,Normal/images/Normal-881.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-881.png,Normal/masks/Normal-881.png,True,False,299,299,L,111.53,73a92744666d79c9638c1c29db63bc3e
Normal-8810.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8810.png,Normal/images/Normal-8810.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8810.png,Normal/masks/Normal-8810.png,True,False,299,299,L,115.26,1de81b71d7f308f5111aa75a609ed7dd
Normal-8811.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8811.png,Normal/images/Normal-8811.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8811.png,Normal/masks/Normal-8811.png,True,False,299,299,L,143.61,682ef3d7bf16c64e5327bdb44232bfab
Normal-8812.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8812.png,Normal/images/Normal-8812.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8812.png,Normal/masks/Normal-8812.png,True,False,299,299,L,142.81,6419ae272ba8068f355c5d89cd5a5ef7
Normal-8813.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8813.png,Normal/images/Normal-8813.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8813.png,Normal/masks/Normal-8813.png,True,False,299,299,L,127.04,5b062a5f1ea094913b26d4819fba57f9
Normal-8814.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8814.png,Normal/images/Normal-8814.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8814.png,Normal/masks/Normal-8814.png,True,False,299,299,L,134.71,6e2d32f3c4eb2f7c154a990132cf585b
Normal-8815.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8815.png,Normal/images/Normal-8815.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8815.png,Normal/masks/Normal-8815.png,True,False,299,299,L,100.34,e888180b44df350fd5e11d5839efee2e
Normal-8816.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8816.png,Normal/images/Normal-8816.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8816.png,Normal/masks/Normal-8816.png,True,False,299,299,L,165.67,04d57d6369272fd8fdfd5b1c6d9c0984
Normal-8817.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8817.png,Normal/images/Normal-8817.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8817.png,Normal/masks/Normal-8817.png,True,False,299,299,L,127.65,f467b426127a2be3180c27dd817077a2
Normal-8818.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8818.png,Normal/images/Normal-8818.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8818.png,Normal/masks/Normal-8818.png,True,False,299,299,L,173.05,0e19cb41cecace88817f242d5b37872d
Normal-8819.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8819.png,Normal/images/Normal-8819.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8819.png,Normal/masks/Normal-8819.png,True,False,299,299,L,119.0,68652a11551e6461e4b9a3c4e0a5bb78
Normal-882.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-882.png,Normal/images/Normal-882.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-882.png,Normal/masks/Normal-882.png,True,False,299,299,L,116.21,ab72adeb75a17679831c1791200a2cb4
Normal-8820.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8820.png,Normal/images/Normal-8820.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8820.png,Normal/masks/Normal-8820.png,True,False,299,299,L,138.0,a2c13e20e2889bae62ca6c9ef7c829af
Normal-8821.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8821.png,Normal/images/Normal-8821.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8821.png,Normal/masks/Normal-8821.png,True,False,299,299,L,124.53,7c8a9930554fda8e435fe8a620455606
Normal-8822.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8822.png,Normal/images/Normal-8822.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8822.png,Normal/masks/Normal-8822.png,True,False,299,299,L,182.36,98785e99c896149871f1cc5b7ef82a73
Normal-8823.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8823.png,Normal/images/Normal-8823.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8823.png,Normal/masks/Normal-8823.png,True,False,299,299,L,158.12,a008fb27ce30ab2d1ea58577ccd90ac3
Normal-8824.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8824.png,Normal/images/Normal-8824.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8824.png,Normal/masks/Normal-8824.png,True,False,299,299,L,126.62,f55a231e80337393e948fbbbea19b039
Normal-8825.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8825.png,Normal/images/Normal-8825.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8825.png,Normal/masks/Normal-8825.png,True,False,299,299,L,166.23,415b2c5ff360ea8703cbed193f6fabdc
Normal-8826.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8826.png,Normal/images/Normal-8826.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8826.png,Normal/masks/Normal-8826.png,True,False,299,299,L,107.39,886a2f92c71ce333c63aba0928fcf3fe
Normal-8827.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8827.png,Normal/images/Normal-8827.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8827.png,Normal/masks/Normal-8827.png,True,False,299,299,L,132.06,77dbdfd80aff87d54999947d1a9f1e62
Normal-8828.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8828.png,Normal/images/Normal-8828.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8828.png,Normal/masks/Normal-8828.png,True,False,299,299,L,120.03,25a3f5a9d5000099ff3b058d4848ad29
Normal-8829.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8829.png,Normal/images/Normal-8829.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8829.png,Normal/masks/Normal-8829.png,True,False,299,299,L,119.57,3c86180d3033618dafcfb0c55f0e0b05
Normal-883.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-883.png,Normal/images/Normal-883.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-883.png,Normal/masks/Normal-883.png,True,False,299,299,L,130.15,53084f699d55c6989058c6bae9c19c5e
Normal-8830.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8830.png,Normal/images/Normal-8830.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8830.png,Normal/masks/Normal-8830.png,True,False,299,299,L,110.63,45366b586852900d23298cbc7e982198
Normal-8831.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8831.png,Normal/images/Normal-8831.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8831.png,Normal/masks/Normal-8831.png,True,False,299,299,L,108.11,351f7908273964e3ed3244f481616ef6
Normal-8832.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8832.png,Normal/images/Normal-8832.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8832.png,Normal/masks/Normal-8832.png,True,False,299,299,L,172.92,29ecf2bef1f6ad6c8482483136c24d0c
Normal-8833.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8833.png,Normal/images/Normal-8833.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8833.png,Normal/masks/Normal-8833.png,True,False,299,299,L,125.84,28e1ee029c7524fbedef92ca50fa0b4a
Normal-8834.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8834.png,Normal/images/Normal-8834.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8834.png,Normal/masks/Normal-8834.png,True,False,299,299,L,149.78,9aef6a06785f3ad04505b3f11cd2cdd0
Normal-8835.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8835.png,Normal/images/Normal-8835.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8835.png,Normal/masks/Normal-8835.png,True,False,299,299,L,94.26,25feec01cc30a07fd769b95a6864f389
Normal-8836.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8836.png,Normal/images/Normal-8836.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8836.png,Normal/masks/Normal-8836.png,True,False,299,299,L,115.94,01e8f5744a21295f41832cbe3debbbb4
Normal-8837.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8837.png,Normal/images/Normal-8837.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8837.png,Normal/masks/Normal-8837.png,True,False,299,299,L,164.13,ab520ea50d00b1c7ca14ac3f3d34eb86
Normal-8838.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8838.png,Normal/images/Normal-8838.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8838.png,Normal/masks/Normal-8838.png,True,False,299,299,L,139.93,6ebcfbdefcbc4ddbef1878dc097b5e7c
Normal-8839.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8839.png,Normal/images/Normal-8839.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8839.png,Normal/masks/Normal-8839.png,True,False,299,299,L,152.22,aaf7e374dcac2d5ebdc202dcb7ab4b03
Normal-884.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-884.png,Normal/images/Normal-884.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-884.png,Normal/masks/Normal-884.png,True,False,299,299,L,118.48,432786996a90f7fce4065adece486deb
Normal-8840.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8840.png,Normal/images/Normal-8840.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8840.png,Normal/masks/Normal-8840.png,True,False,299,299,L,114.21,062502f5bb146ae391ca7564967b716f
Normal-8841.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8841.png,Normal/images/Normal-8841.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8841.png,Normal/masks/Normal-8841.png,True,False,299,299,L,162.33,5fdd33285293b7f38e05f43e53ef2a11
Normal-8842.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8842.png,Normal/images/Normal-8842.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8842.png,Normal/masks/Normal-8842.png,True,False,299,299,L,132.05,7370b2369e85faeeded9bab543d0043d
Normal-8843.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8843.png,Normal/images/Normal-8843.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8843.png,Normal/masks/Normal-8843.png,True,False,299,299,L,124.33,fdac09e3c345773f896544e109945b55
Normal-8844.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8844.png,Normal/images/Normal-8844.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8844.png,Normal/masks/Normal-8844.png,True,False,299,299,L,87.47,a063979f06f74d1902ae781788e8d123
Normal-8845.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8845.png,Normal/images/Normal-8845.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8845.png,Normal/masks/Normal-8845.png,True,False,299,299,L,103.99,bcbf7ea2c9e27e475472b55f84866565
Normal-8846.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8846.png,Normal/images/Normal-8846.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8846.png,Normal/masks/Normal-8846.png,True,False,299,299,L,115.43,b82524151ad158f2ecae873f110862ec
Normal-8847.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8847.png,Normal/images/Normal-8847.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8847.png,Normal/masks/Normal-8847.png,True,False,299,299,L,129.42,27146ad36d55ef8cc32ac223cc2e390a
Normal-8848.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8848.png,Normal/images/Normal-8848.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8848.png,Normal/masks/Normal-8848.png,True,False,299,299,L,110.6,65d7ab0abd6a3d5d2787b3f7c0f4ba69
Normal-8849.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8849.png,Normal/images/Normal-8849.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8849.png,Normal/masks/Normal-8849.png,True,False,299,299,L,155.33,1185bd14cc6718ca862687e207aaa6dc
Normal-885.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-885.png,Normal/images/Normal-885.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-885.png,Normal/masks/Normal-885.png,True,False,299,299,L,156.25,a255a0c7024cb23e42c3a3894816fa00
Normal-8850.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8850.png,Normal/images/Normal-8850.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8850.png,Normal/masks/Normal-8850.png,True,False,299,299,L,169.41,c4cd22fbc2a7c3186596e38aea564273
Normal-8851.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8851.png,Normal/images/Normal-8851.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8851.png,Normal/masks/Normal-8851.png,True,False,299,299,L,120.46,0f346a21ccc957d6c78d3ce54ef0e92e
Normal-8852.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8852.png,Normal/images/Normal-8852.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8852.png,Normal/masks/Normal-8852.png,True,False,299,299,L,130.62,2107a0a57d3e4edc1306078342a65273
Normal-8853.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8853.png,Normal/images/Normal-8853.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8853.png,Normal/masks/Normal-8853.png,True,False,299,299,L,107.24,341961df1d9f3942db9619789fcce935
Normal-8854.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8854.png,Normal/images/Normal-8854.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8854.png,Normal/masks/Normal-8854.png,True,False,299,299,L,177.35,8dcdc0fd64af3df312c4ba32681d9b12
Normal-8855.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8855.png,Normal/images/Normal-8855.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8855.png,Normal/masks/Normal-8855.png,True,False,299,299,L,168.01,c715d5c00a4236a6d8a34d0a7abdf306
Normal-8856.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8856.png,Normal/images/Normal-8856.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8856.png,Normal/masks/Normal-8856.png,True,False,299,299,L,144.95,e0d8567c3e43707fd72030e75ee9a9f6
Normal-8857.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8857.png,Normal/images/Normal-8857.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8857.png,Normal/masks/Normal-8857.png,True,False,299,299,L,132.67,583e347ecf7405d3a884adac3943b28d
Normal-8858.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8858.png,Normal/images/Normal-8858.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8858.png,Normal/masks/Normal-8858.png,True,False,299,299,L,123.55,c938cc83c6ef04c550c9cb9c19a960b2
Normal-8859.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8859.png,Normal/images/Normal-8859.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8859.png,Normal/masks/Normal-8859.png,True,False,299,299,L,139.09,8b6ebdadaf23cfdbed616b0ff76d9d19
Normal-886.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-886.png,Normal/images/Normal-886.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-886.png,Normal/masks/Normal-886.png,True,False,299,299,L,118.44,b525e260f520fde475130e99924f4a6b
Normal-8860.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8860.png,Normal/images/Normal-8860.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8860.png,Normal/masks/Normal-8860.png,True,False,299,299,L,111.55,7c809e69fa0f80f0ded89684be9f2883
Normal-8861.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8861.png,Normal/images/Normal-8861.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8861.png,Normal/masks/Normal-8861.png,True,False,299,299,L,103.77,1037ee8bff45814f59f250d595647f16
Normal-8862.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8862.png,Normal/images/Normal-8862.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8862.png,Normal/masks/Normal-8862.png,True,False,299,299,L,171.48,014577397927688ff8876e5c6066be66
Normal-8863.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8863.png,Normal/images/Normal-8863.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8863.png,Normal/masks/Normal-8863.png,True,False,299,299,L,157.37,a552074df3f65f7d3b09869af5a1ce68
Normal-8864.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8864.png,Normal/images/Normal-8864.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8864.png,Normal/masks/Normal-8864.png,True,False,299,299,L,121.91,649b42528e97b659f38f4d1855945c9b
Normal-8865.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8865.png,Normal/images/Normal-8865.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8865.png,Normal/masks/Normal-8865.png,True,False,299,299,L,155.76,f7ca840a15f52dfc7cb1fdd1b1cdee0f
Normal-8866.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8866.png,Normal/images/Normal-8866.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8866.png,Normal/masks/Normal-8866.png,True,False,299,299,L,167.1,037d47691b28244f6e527ae68678a5f2
Normal-8867.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8867.png,Normal/images/Normal-8867.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8867.png,Normal/masks/Normal-8867.png,True,False,299,299,L,154.4,d101ecd0953b97736341879610037871
Normal-8868.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8868.png,Normal/images/Normal-8868.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8868.png,Normal/masks/Normal-8868.png,True,False,299,299,L,165.55,c10729d053899b1874a2c04a1c734ded
Normal-8869.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8869.png,Normal/images/Normal-8869.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8869.png,Normal/masks/Normal-8869.png,True,False,299,299,L,107.79,beea68e995ef45e3ccfa4f3f39b68f71
Normal-887.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-887.png,Normal/images/Normal-887.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-887.png,Normal/masks/Normal-887.png,True,False,299,299,L,114.23,167843db9fcd520f73a70124d0a8fe4f
Normal-8870.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8870.png,Normal/images/Normal-8870.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8870.png,Normal/masks/Normal-8870.png,True,False,299,299,L,141.49,61c437e542e3785951d5e1683f3b6758
Normal-8871.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8871.png,Normal/images/Normal-8871.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8871.png,Normal/masks/Normal-8871.png,True,False,299,299,L,132.69,6761c2a3a306a0f2b0c752169188ce4a
Normal-8872.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8872.png,Normal/images/Normal-8872.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8872.png,Normal/masks/Normal-8872.png,True,False,299,299,L,129.05,4f314cbf8f3951140434d544dc6aea29
Normal-8873.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8873.png,Normal/images/Normal-8873.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8873.png,Normal/masks/Normal-8873.png,True,False,299,299,L,129.89,894775b9495f6e4ac33194b39eb60882
Normal-8874.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8874.png,Normal/images/Normal-8874.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8874.png,Normal/masks/Normal-8874.png,True,False,299,299,L,174.51,70e2a213b0c806b3b39e6a460eb31d56
Normal-8875.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8875.png,Normal/images/Normal-8875.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8875.png,Normal/masks/Normal-8875.png,True,False,299,299,L,123.38,55171d827e17bf83e5f4be1f1004a47b
Normal-8876.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8876.png,Normal/images/Normal-8876.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8876.png,Normal/masks/Normal-8876.png,True,False,299,299,L,123.07,ec0b3706c056fd0aeceb5273abfd8a59
Normal-8877.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8877.png,Normal/images/Normal-8877.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8877.png,Normal/masks/Normal-8877.png,True,False,299,299,L,153.83,ca456644acb4e7b4703dfc2ce17423a3
Normal-8878.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8878.png,Normal/images/Normal-8878.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8878.png,Normal/masks/Normal-8878.png,True,False,299,299,L,113.44,daac54b5dd197b0141deb1a1fb504a10
Normal-8879.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8879.png,Normal/images/Normal-8879.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8879.png,Normal/masks/Normal-8879.png,True,False,299,299,L,149.63,ff8ca4d22c31b614ba7b00a2e78e7c29
Normal-888.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-888.png,Normal/images/Normal-888.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-888.png,Normal/masks/Normal-888.png,True,False,299,299,L,114.55,4d7c59ff0d7b273f9d87aaf3aec23cb9
Normal-8880.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8880.png,Normal/images/Normal-8880.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8880.png,Normal/masks/Normal-8880.png,True,False,299,299,L,146.33,6a5c6c268b7d8c745673390171c6138a
Normal-8881.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8881.png,Normal/images/Normal-8881.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8881.png,Normal/masks/Normal-8881.png,True,False,299,299,L,113.32,a6566aae811d0be789e3a2a5635c2031
Normal-8882.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8882.png,Normal/images/Normal-8882.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8882.png,Normal/masks/Normal-8882.png,True,False,299,299,L,148.04,1417cdaf58bbd1a5ec59a966649752cf
Normal-8883.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8883.png,Normal/images/Normal-8883.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8883.png,Normal/masks/Normal-8883.png,True,False,299,299,L,124.93,5ebb39f57471015ba5d80ec000b425f0
Normal-8884.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8884.png,Normal/images/Normal-8884.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8884.png,Normal/masks/Normal-8884.png,True,False,299,299,L,117.76,7f53ff99ea0adb099e853f4388841cea
Normal-8885.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8885.png,Normal/images/Normal-8885.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8885.png,Normal/masks/Normal-8885.png,True,False,299,299,L,117.55,1138f5ce92308cc7bfc9a21e5e6044db
Normal-8886.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8886.png,Normal/images/Normal-8886.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8886.png,Normal/masks/Normal-8886.png,True,False,299,299,L,112.71,164eb594e5d4640b74b5a9e4ea2b1762
Normal-8887.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8887.png,Normal/images/Normal-8887.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8887.png,Normal/masks/Normal-8887.png,True,False,299,299,L,144.73,324482642f3f86190e45c16bd4b26060
Normal-8888.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8888.png,Normal/images/Normal-8888.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8888.png,Normal/masks/Normal-8888.png,True,False,299,299,L,137.93,ebd602cb1f0a8ca026e6ea33750f7146
Normal-8889.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8889.png,Normal/images/Normal-8889.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8889.png,Normal/masks/Normal-8889.png,True,False,299,299,L,125.56,25d37019ee20bed5647cf08a1f38cb53
Normal-889.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-889.png,Normal/images/Normal-889.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-889.png,Normal/masks/Normal-889.png,True,False,299,299,L,118.87,794de035562d847ef6433661f512efe6
Normal-8890.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8890.png,Normal/images/Normal-8890.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8890.png,Normal/masks/Normal-8890.png,True,False,299,299,L,93.86,9dd962e8f7ed2bb7a253292a6a6087e0
Normal-8891.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8891.png,Normal/images/Normal-8891.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8891.png,Normal/masks/Normal-8891.png,True,False,299,299,L,113.15,0ff233397ad9be4cd046bfba5dece72c
Normal-8892.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8892.png,Normal/images/Normal-8892.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8892.png,Normal/masks/Normal-8892.png,True,False,299,299,L,164.79,803d92bd07303ff79e545b2ec8d4b83b
Normal-8893.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8893.png,Normal/images/Normal-8893.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8893.png,Normal/masks/Normal-8893.png,True,False,299,299,L,131.36,e982760e3f2988779edc4afbc08fbc5d
Normal-8894.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8894.png,Normal/images/Normal-8894.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8894.png,Normal/masks/Normal-8894.png,True,False,299,299,L,154.17,c59616da50c1346d1ce3c1895dd1d168
Normal-8895.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8895.png,Normal/images/Normal-8895.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8895.png,Normal/masks/Normal-8895.png,True,False,299,299,L,113.87,7682c2311e649ac27d594acf9b362f3e
Normal-8896.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8896.png,Normal/images/Normal-8896.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8896.png,Normal/masks/Normal-8896.png,True,False,299,299,L,135.72,165fa3600321428a20d3110ee5b7bada
Normal-8897.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8897.png,Normal/images/Normal-8897.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8897.png,Normal/masks/Normal-8897.png,True,False,299,299,L,120.25,fe7155d5f60d311256653a5b2ecf7c34
Normal-8898.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8898.png,Normal/images/Normal-8898.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8898.png,Normal/masks/Normal-8898.png,True,False,299,299,L,126.73,7e7b6f4b38ca658dae7c64d7b8f5178f
Normal-8899.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8899.png,Normal/images/Normal-8899.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8899.png,Normal/masks/Normal-8899.png,True,False,299,299,L,145.85,750cb1ec750fc70e04aa71e0421a8009
Normal-89.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-89.png,Normal/images/Normal-89.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-89.png,Normal/masks/Normal-89.png,True,False,299,299,L,118.48,0c921496d974ff870ec9ca499684d7c5
Normal-890.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-890.png,Normal/images/Normal-890.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-890.png,Normal/masks/Normal-890.png,True,False,299,299,L,115.71,65fbef21acca2c84711e65062c87ac9b
Normal-8900.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8900.png,Normal/images/Normal-8900.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8900.png,Normal/masks/Normal-8900.png,True,False,299,299,L,170.6,16c50523f3142e0120d4cf90c0a470c6
Normal-8901.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8901.png,Normal/images/Normal-8901.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8901.png,Normal/masks/Normal-8901.png,True,False,299,299,L,115.79,18996b966b6d1f5fc04d0dfff6bb289e
Normal-8902.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8902.png,Normal/images/Normal-8902.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8902.png,Normal/masks/Normal-8902.png,True,False,299,299,L,99.26,da10497ec3b0acaff66f074d70d27bbd
Normal-8903.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8903.png,Normal/images/Normal-8903.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8903.png,Normal/masks/Normal-8903.png,True,False,299,299,L,170.76,fdf2390afd287b0462984d8146591c55
Normal-8904.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8904.png,Normal/images/Normal-8904.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8904.png,Normal/masks/Normal-8904.png,True,False,299,299,L,92.83,b9fc8396983b751cd85e56d6664dc411
Normal-8905.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8905.png,Normal/images/Normal-8905.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8905.png,Normal/masks/Normal-8905.png,True,False,299,299,L,100.2,11bd59d60d5465873cb93ffdc0adf158
Normal-8906.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8906.png,Normal/images/Normal-8906.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8906.png,Normal/masks/Normal-8906.png,True,False,299,299,L,121.56,955e552b641b30dfd3e6a8585fe7f898
Normal-8907.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8907.png,Normal/images/Normal-8907.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8907.png,Normal/masks/Normal-8907.png,True,False,299,299,L,168.39,54737c80ab8b647fed99ca53a2581de8
Normal-8908.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8908.png,Normal/images/Normal-8908.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8908.png,Normal/masks/Normal-8908.png,True,False,299,299,L,122.67,1cd1f12047d11be36013182e70ce6adb
Normal-8909.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8909.png,Normal/images/Normal-8909.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8909.png,Normal/masks/Normal-8909.png,True,False,299,299,L,105.73,0c8b568ef6a0c229d4971216f9288cc1
Normal-891.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-891.png,Normal/images/Normal-891.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-891.png,Normal/masks/Normal-891.png,True,False,299,299,L,127.63,389055d885871d247ce2d3c5c6f71b8b
Normal-8910.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8910.png,Normal/images/Normal-8910.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8910.png,Normal/masks/Normal-8910.png,True,False,299,299,L,144.78,c2f8bc12ce6d2bb6b81080b1617ccbed
Normal-8911.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8911.png,Normal/images/Normal-8911.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8911.png,Normal/masks/Normal-8911.png,True,False,299,299,L,109.67,55ea5f4dea38140fc855364c0036d235
Normal-8912.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8912.png,Normal/images/Normal-8912.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8912.png,Normal/masks/Normal-8912.png,True,False,299,299,L,127.38,18123f8da3126a0f47d5b0d83f752911
Normal-8913.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8913.png,Normal/images/Normal-8913.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8913.png,Normal/masks/Normal-8913.png,True,False,299,299,L,144.57,72ce9766c7fcff35a30ba6b21647971e
Normal-8914.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8914.png,Normal/images/Normal-8914.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8914.png,Normal/masks/Normal-8914.png,True,False,299,299,L,151.48,c82f535bdb1140f1aab20c6a7a712f25
Normal-8915.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8915.png,Normal/images/Normal-8915.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8915.png,Normal/masks/Normal-8915.png,True,False,299,299,L,131.09,5c235ea80e66ab046735f7aeefa62d11
Normal-8916.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8916.png,Normal/images/Normal-8916.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8916.png,Normal/masks/Normal-8916.png,True,False,299,299,L,142.68,da987e2bf156f595916c82b9be6f013a
Normal-8917.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8917.png,Normal/images/Normal-8917.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8917.png,Normal/masks/Normal-8917.png,True,False,299,299,L,106.82,b640c1e9cca33ddf1ab796eb11ef54d4
Normal-8918.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8918.png,Normal/images/Normal-8918.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8918.png,Normal/masks/Normal-8918.png,True,False,299,299,L,127.08,4bc7c17d373221131365ffb011bb4e09
Normal-8919.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8919.png,Normal/images/Normal-8919.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8919.png,Normal/masks/Normal-8919.png,True,False,299,299,L,140.49,80c57853ad3925ee2f4b0da0c2e2aecf
Normal-892.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-892.png,Normal/images/Normal-892.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-892.png,Normal/masks/Normal-892.png,True,False,299,299,L,135.17,964ebcde695956a821bf286b2819f310
Normal-8920.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8920.png,Normal/images/Normal-8920.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8920.png,Normal/masks/Normal-8920.png,True,False,299,299,L,143.87,093577397249d88241f51c41042287fd
Normal-8921.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8921.png,Normal/images/Normal-8921.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8921.png,Normal/masks/Normal-8921.png,True,False,299,299,L,124.54,3bee3a7bcb845a19fe451c72557b2646
Normal-8922.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8922.png,Normal/images/Normal-8922.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8922.png,Normal/masks/Normal-8922.png,True,False,299,299,L,132.01,86c3fac4ec2d2be64c737b7df824b45f
Normal-8923.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8923.png,Normal/images/Normal-8923.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8923.png,Normal/masks/Normal-8923.png,True,False,299,299,L,120.21,2d65dbc48d394fac6e816f2f50e3b5bf
Normal-8924.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8924.png,Normal/images/Normal-8924.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8924.png,Normal/masks/Normal-8924.png,True,False,299,299,L,124.0,49ab18fda1fae0308ab02d1d47d69ea2
Normal-8925.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8925.png,Normal/images/Normal-8925.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8925.png,Normal/masks/Normal-8925.png,True,False,299,299,L,82.86,d8d729a1b1f1a58e915bb0bc54438fc4
Normal-8926.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8926.png,Normal/images/Normal-8926.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8926.png,Normal/masks/Normal-8926.png,True,False,299,299,L,123.68,0962a19244f4c05f424268f4478da801
Normal-8927.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8927.png,Normal/images/Normal-8927.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8927.png,Normal/masks/Normal-8927.png,True,False,299,299,L,154.96,e650fff1ce5f62483745b2b23234caee
Normal-8928.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8928.png,Normal/images/Normal-8928.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8928.png,Normal/masks/Normal-8928.png,True,False,299,299,L,145.58,1cd4ae246bfd2ed1edd271e77eda12e0
Normal-8929.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8929.png,Normal/images/Normal-8929.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8929.png,Normal/masks/Normal-8929.png,True,False,299,299,L,108.74,5c745d905fddcac6091cd925d34be3fe
Normal-893.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-893.png,Normal/images/Normal-893.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-893.png,Normal/masks/Normal-893.png,True,False,299,299,L,125.21,fcd2daf5a21b66a43b30807480987e7d
Normal-8930.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8930.png,Normal/images/Normal-8930.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8930.png,Normal/masks/Normal-8930.png,True,False,299,299,L,156.44,18a101f54fbc6e4198d8cbcba9c57aad
Normal-8931.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8931.png,Normal/images/Normal-8931.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8931.png,Normal/masks/Normal-8931.png,True,False,299,299,L,128.55,dca2a5bedf6ecdb437834e3e9b3a8a45
Normal-8932.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8932.png,Normal/images/Normal-8932.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8932.png,Normal/masks/Normal-8932.png,True,False,299,299,L,106.77,830e07b215e92715e89f4e911a37dc88
Normal-8933.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8933.png,Normal/images/Normal-8933.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8933.png,Normal/masks/Normal-8933.png,True,False,299,299,L,127.21,9d7d6e99b3b697752e6d8013167a8166
Normal-8934.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8934.png,Normal/images/Normal-8934.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8934.png,Normal/masks/Normal-8934.png,True,False,299,299,L,103.48,b3277aecfc01cb93dbb4c6a8da8a66ae
Normal-8935.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8935.png,Normal/images/Normal-8935.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8935.png,Normal/masks/Normal-8935.png,True,False,299,299,L,175.09,27e43522cef5267961eba6863ba43abc
Normal-8936.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8936.png,Normal/images/Normal-8936.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8936.png,Normal/masks/Normal-8936.png,True,False,299,299,L,152.75,3eafc5f09d3efecd27c48f2b3e8fc265
Normal-8937.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8937.png,Normal/images/Normal-8937.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8937.png,Normal/masks/Normal-8937.png,True,False,299,299,L,97.15,f7f3f39ca249d6ed049713eb39a3022d
Normal-8938.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8938.png,Normal/images/Normal-8938.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8938.png,Normal/masks/Normal-8938.png,True,False,299,299,L,100.32,af5c201b7603227f25fe193462472d5f
Normal-8939.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8939.png,Normal/images/Normal-8939.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8939.png,Normal/masks/Normal-8939.png,True,False,299,299,L,181.67,13c157bed61cd27cd8491e9f46c1ee44
Normal-894.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-894.png,Normal/images/Normal-894.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-894.png,Normal/masks/Normal-894.png,True,False,299,299,L,124.4,a37401a3e84634fce208f417294fe3a9
Normal-8940.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8940.png,Normal/images/Normal-8940.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8940.png,Normal/masks/Normal-8940.png,True,False,299,299,L,85.39,a1a0111858b0a54318830062eabd49ea
Normal-8941.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8941.png,Normal/images/Normal-8941.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8941.png,Normal/masks/Normal-8941.png,True,False,299,299,L,172.98,809992e5a141144252112f9bba561bc7
Normal-8942.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8942.png,Normal/images/Normal-8942.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8942.png,Normal/masks/Normal-8942.png,True,False,299,299,L,159.08,71dec210b1c6abdfab047132c7ae759f
Normal-8943.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8943.png,Normal/images/Normal-8943.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8943.png,Normal/masks/Normal-8943.png,True,False,299,299,L,119.7,42a7f389dca17a9b8ca21270d069cea8
Normal-8944.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8944.png,Normal/images/Normal-8944.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8944.png,Normal/masks/Normal-8944.png,True,False,299,299,L,152.65,3da149020020e981e1dc8542423683ee
Normal-8945.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8945.png,Normal/images/Normal-8945.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8945.png,Normal/masks/Normal-8945.png,True,False,299,299,L,178.96,27612473a46a8ea1477ca81639313f59
Normal-8946.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8946.png,Normal/images/Normal-8946.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8946.png,Normal/masks/Normal-8946.png,True,False,299,299,L,123.6,f0e85c7cd988083d41d386478f924a39
Normal-8947.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8947.png,Normal/images/Normal-8947.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8947.png,Normal/masks/Normal-8947.png,True,False,299,299,L,109.86,8f5253eb37ca49099f27fce5e4c3d3b3
Normal-8948.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8948.png,Normal/images/Normal-8948.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8948.png,Normal/masks/Normal-8948.png,True,False,299,299,L,175.76,5a53ad74d7f7e36f7fe893443ab83a7d
Normal-8949.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8949.png,Normal/images/Normal-8949.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8949.png,Normal/masks/Normal-8949.png,True,False,299,299,L,112.44,8d577c314f4ca1447a47bbac3cfe2665
Normal-895.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-895.png,Normal/images/Normal-895.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-895.png,Normal/masks/Normal-895.png,True,False,299,299,L,123.7,24d09278bc3a0f60b617e540472df8dd
Normal-8950.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8950.png,Normal/images/Normal-8950.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8950.png,Normal/masks/Normal-8950.png,True,False,299,299,L,106.9,4da59f2a0d0b7719d67baaefbc1c6bbc
Normal-8951.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8951.png,Normal/images/Normal-8951.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8951.png,Normal/masks/Normal-8951.png,True,False,299,299,L,142.33,bf75c3b5ff3783e3fa94b951ce3cbc1a
Normal-8952.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8952.png,Normal/images/Normal-8952.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8952.png,Normal/masks/Normal-8952.png,True,False,299,299,L,127.14,50f79f36d29fcfa8dd6ffc703e456e53
Normal-8953.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8953.png,Normal/images/Normal-8953.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8953.png,Normal/masks/Normal-8953.png,True,False,299,299,L,144.47,f179c9413f972ad9209ecbb87273f9a0
Normal-8954.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8954.png,Normal/images/Normal-8954.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8954.png,Normal/masks/Normal-8954.png,True,False,299,299,L,127.37,ce2d05f91c1e4fc3c483900bdde85812
Normal-8955.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8955.png,Normal/images/Normal-8955.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8955.png,Normal/masks/Normal-8955.png,True,False,299,299,L,104.03,8400cdc21d02e06604d5383c8667293b
Normal-8956.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8956.png,Normal/images/Normal-8956.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8956.png,Normal/masks/Normal-8956.png,True,False,299,299,L,103.98,fcfeeb3697929fc677a6601fdd84df8c
Normal-8957.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8957.png,Normal/images/Normal-8957.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8957.png,Normal/masks/Normal-8957.png,True,False,299,299,L,117.38,90b8d633dfe5a963ccf1b3bcb75a6e46
Normal-8958.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8958.png,Normal/images/Normal-8958.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8958.png,Normal/masks/Normal-8958.png,True,False,299,299,L,119.9,c3920cf52ddebb8b7c4b9faae172501a
Normal-8959.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8959.png,Normal/images/Normal-8959.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8959.png,Normal/masks/Normal-8959.png,True,False,299,299,L,164.9,c48654e6c8fe0d37fa8b50c4449fa098
Normal-896.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-896.png,Normal/images/Normal-896.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-896.png,Normal/masks/Normal-896.png,True,False,299,299,L,122.39,b26a0625de48d8df5e212ea2e52bee75
Normal-8960.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8960.png,Normal/images/Normal-8960.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8960.png,Normal/masks/Normal-8960.png,True,False,299,299,L,110.11,8176c86d307cde43feb793029d0258fb
Normal-8961.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8961.png,Normal/images/Normal-8961.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8961.png,Normal/masks/Normal-8961.png,True,False,299,299,L,114.0,5dd9a72a370ecfa298bc1cc3fac5fd6d
Normal-8962.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8962.png,Normal/images/Normal-8962.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8962.png,Normal/masks/Normal-8962.png,True,False,299,299,L,146.75,1f65066d561730764b8edd85d4cbf6a5
Normal-8963.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8963.png,Normal/images/Normal-8963.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8963.png,Normal/masks/Normal-8963.png,True,False,299,299,L,140.99,57c9f07ab3a7cb946425fd727dedd35b
Normal-8964.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8964.png,Normal/images/Normal-8964.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8964.png,Normal/masks/Normal-8964.png,True,False,299,299,L,82.22,f237a20143fcd95ca536b678cf29288c
Normal-8965.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8965.png,Normal/images/Normal-8965.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8965.png,Normal/masks/Normal-8965.png,True,False,299,299,L,128.94,fccc29c2341bd629722162e358c23312
Normal-8966.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8966.png,Normal/images/Normal-8966.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8966.png,Normal/masks/Normal-8966.png,True,False,299,299,L,104.41,3c5b821d81b6e7517aaa3a6cd4bf7c2d
Normal-8967.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8967.png,Normal/images/Normal-8967.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8967.png,Normal/masks/Normal-8967.png,True,False,299,299,L,166.56,654ebe738d6b3bd773ac11d41af03a83
Normal-8968.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8968.png,Normal/images/Normal-8968.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8968.png,Normal/masks/Normal-8968.png,True,False,299,299,L,98.89,30a7d76b0680c7d8acf9185a40c1b67b
Normal-8969.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8969.png,Normal/images/Normal-8969.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8969.png,Normal/masks/Normal-8969.png,True,False,299,299,L,112.55,57b426b6f31834a11f4a2eb885479d53
Normal-897.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-897.png,Normal/images/Normal-897.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-897.png,Normal/masks/Normal-897.png,True,False,299,299,L,119.44,bcd2335fff50005f32f1af63408c4f2a
Normal-8970.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8970.png,Normal/images/Normal-8970.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8970.png,Normal/masks/Normal-8970.png,True,False,299,299,L,112.55,229a11a7a0b73c329659ec2cfe6f4002
Normal-8971.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8971.png,Normal/images/Normal-8971.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8971.png,Normal/masks/Normal-8971.png,True,False,299,299,L,116.42,cce999ea12921aef6b65009ff57eaf05
Normal-8972.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8972.png,Normal/images/Normal-8972.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8972.png,Normal/masks/Normal-8972.png,True,False,299,299,L,107.64,5aafbe8be266cd80e2b26cbde909ba74
Normal-8973.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8973.png,Normal/images/Normal-8973.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8973.png,Normal/masks/Normal-8973.png,True,False,299,299,L,126.17,e33bd1698efcd319103e40395686f122
Normal-8974.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8974.png,Normal/images/Normal-8974.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8974.png,Normal/masks/Normal-8974.png,True,False,299,299,L,111.01,1ea095d9d7faee8b257d2d4334293a21
Normal-8975.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8975.png,Normal/images/Normal-8975.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8975.png,Normal/masks/Normal-8975.png,True,False,299,299,L,187.66,1a907a8a6c75e86b5f8bc0dd1d0859a3
Normal-8976.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8976.png,Normal/images/Normal-8976.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8976.png,Normal/masks/Normal-8976.png,True,False,299,299,L,96.57,1f7f8593866ad244dad3202adbf8d603
Normal-8977.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8977.png,Normal/images/Normal-8977.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8977.png,Normal/masks/Normal-8977.png,True,False,299,299,L,107.41,a871525aef94b5af7458e42d8f97b515
Normal-8978.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8978.png,Normal/images/Normal-8978.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8978.png,Normal/masks/Normal-8978.png,True,False,299,299,L,132.76,eaea083e80fd5d0b414c87a2fd06b288
Normal-8979.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8979.png,Normal/images/Normal-8979.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8979.png,Normal/masks/Normal-8979.png,True,False,299,299,L,131.26,7dedcbfcb7e90a801737658ba1ea2231
Normal-898.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-898.png,Normal/images/Normal-898.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-898.png,Normal/masks/Normal-898.png,True,False,299,299,L,124.55,27202f506855982b658397b7d07ac4f7
Normal-8980.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8980.png,Normal/images/Normal-8980.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8980.png,Normal/masks/Normal-8980.png,True,False,299,299,L,155.65,96ce73f2d9ac4a5298f50c6e255da73b
Normal-8981.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8981.png,Normal/images/Normal-8981.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8981.png,Normal/masks/Normal-8981.png,True,False,299,299,L,153.67,c772ba200a695c214e24e50b0a88d522
Normal-8982.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8982.png,Normal/images/Normal-8982.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8982.png,Normal/masks/Normal-8982.png,True,False,299,299,L,100.23,94469ec4ad7d1d0370b46e734ef59eb4
Normal-8983.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8983.png,Normal/images/Normal-8983.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8983.png,Normal/masks/Normal-8983.png,True,False,299,299,L,107.69,90f53997f8c4f4d2411c1fdc13fbbfcd
Normal-8984.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8984.png,Normal/images/Normal-8984.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8984.png,Normal/masks/Normal-8984.png,True,False,299,299,L,114.14,9600e1e5641e271314ca1a3283686c86
Normal-8985.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8985.png,Normal/images/Normal-8985.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8985.png,Normal/masks/Normal-8985.png,True,False,299,299,L,119.58,d4009c75f1998bd081dca8f937cdfa31
Normal-8986.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8986.png,Normal/images/Normal-8986.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8986.png,Normal/masks/Normal-8986.png,True,False,299,299,L,125.61,d0f04d12c6f4f0e8deef68ce2259c6e1
Normal-8987.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8987.png,Normal/images/Normal-8987.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8987.png,Normal/masks/Normal-8987.png,True,False,299,299,L,107.47,34dd5b1e9254d4aa8143262f7367b5f5
Normal-8988.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8988.png,Normal/images/Normal-8988.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8988.png,Normal/masks/Normal-8988.png,True,False,299,299,L,178.26,aecf4e8b496edcf0ac3070ccae8628c7
Normal-8989.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8989.png,Normal/images/Normal-8989.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8989.png,Normal/masks/Normal-8989.png,True,False,299,299,L,166.79,092beaa6223d6cf8d7bb835d19379fbb
Normal-899.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-899.png,Normal/images/Normal-899.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-899.png,Normal/masks/Normal-899.png,True,False,299,299,L,126.19,2f719862fc6e566bfe72e1af9378a2ca
Normal-8990.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8990.png,Normal/images/Normal-8990.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8990.png,Normal/masks/Normal-8990.png,True,False,299,299,L,152.63,811bf4005200ad3964e996729035e7af
Normal-8991.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8991.png,Normal/images/Normal-8991.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8991.png,Normal/masks/Normal-8991.png,True,False,299,299,L,180.96,185c96f68d8371a67ab78c244c50b3ab
Normal-8992.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8992.png,Normal/images/Normal-8992.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8992.png,Normal/masks/Normal-8992.png,True,False,299,299,L,159.97,1549ac145e36da0784c689635ae9ea24
Normal-8993.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8993.png,Normal/images/Normal-8993.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8993.png,Normal/masks/Normal-8993.png,True,False,299,299,L,107.63,f56a2b448240b8c46b3b831a84130323
Normal-8994.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8994.png,Normal/images/Normal-8994.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8994.png,Normal/masks/Normal-8994.png,True,False,299,299,L,126.63,af54ea9bab04f71b0c067a0f4cb40a7b
Normal-8995.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8995.png,Normal/images/Normal-8995.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8995.png,Normal/masks/Normal-8995.png,True,False,299,299,L,127.95,8d69705f2426342fb430032fab5b408d
Normal-8996.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8996.png,Normal/images/Normal-8996.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8996.png,Normal/masks/Normal-8996.png,True,False,299,299,L,125.92,ee640a26c3cb6e11728ebf2291a6272e
Normal-8997.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8997.png,Normal/images/Normal-8997.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8997.png,Normal/masks/Normal-8997.png,True,False,299,299,L,104.65,d2e55ba616eeff15835506a7f9ee94ec
Normal-8998.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8998.png,Normal/images/Normal-8998.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8998.png,Normal/masks/Normal-8998.png,True,False,299,299,L,120.07,568c7e2c9f0a9b2137c28b13b45a953a
Normal-8999.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-8999.png,Normal/images/Normal-8999.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-8999.png,Normal/masks/Normal-8999.png,True,False,299,299,L,145.31,4066ae41e4838f9a89dc971197991142
Normal-9.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9.png,Normal/images/Normal-9.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9.png,Normal/masks/Normal-9.png,True,False,299,299,L,104.64,8d08fe25cd6032d5a452c683a5c9b599
Normal-90.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-90.png,Normal/images/Normal-90.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-90.png,Normal/masks/Normal-90.png,True,False,299,299,L,119.24,98ff7d775e787f4339ade2db54e3b4d6
Normal-900.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-900.png,Normal/images/Normal-900.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-900.png,Normal/masks/Normal-900.png,True,False,299,299,L,168.38,e16dc954fb5bb9fd1863273aa0ea5d70
Normal-9000.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9000.png,Normal/images/Normal-9000.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9000.png,Normal/masks/Normal-9000.png,True,False,299,299,L,132.06,49db780c36a8014b83ce8da47d0658b6
Normal-9001.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9001.png,Normal/images/Normal-9001.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9001.png,Normal/masks/Normal-9001.png,True,False,299,299,L,107.25,49c1a68dc43d50fb7cf2d0ad75d75933
Normal-9002.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9002.png,Normal/images/Normal-9002.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9002.png,Normal/masks/Normal-9002.png,True,False,299,299,L,163.19,7a5cf058b44d42e1161da2586fed8e10
Normal-9003.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9003.png,Normal/images/Normal-9003.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9003.png,Normal/masks/Normal-9003.png,True,False,299,299,L,137.01,189175ca7c5ea3b72431bcc500d01b77
Normal-9004.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9004.png,Normal/images/Normal-9004.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9004.png,Normal/masks/Normal-9004.png,True,False,299,299,L,127.2,3ffcee61d8b8c727a4832b2d559798e9
Normal-9005.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9005.png,Normal/images/Normal-9005.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9005.png,Normal/masks/Normal-9005.png,True,False,299,299,L,107.19,2d059eeeeefec5965dd129008bc0ca94
Normal-9006.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9006.png,Normal/images/Normal-9006.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9006.png,Normal/masks/Normal-9006.png,True,False,299,299,L,111.74,3df1fccc2dd9fd29ac98448007f929d8
Normal-9007.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9007.png,Normal/images/Normal-9007.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9007.png,Normal/masks/Normal-9007.png,True,False,299,299,L,114.9,60197ca2cf3e3b2ac800b681373f5f9f
Normal-9008.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9008.png,Normal/images/Normal-9008.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9008.png,Normal/masks/Normal-9008.png,True,False,299,299,L,117.26,601b5da8365033210a401e18d2e9b290
Normal-9009.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9009.png,Normal/images/Normal-9009.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9009.png,Normal/masks/Normal-9009.png,True,False,299,299,L,161.89,cd35cba96133806c34d266e123df0860
Normal-901.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-901.png,Normal/images/Normal-901.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-901.png,Normal/masks/Normal-901.png,True,False,299,299,L,158.32,59aff61c683aff6c48586df502d423ac
Normal-9010.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9010.png,Normal/images/Normal-9010.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9010.png,Normal/masks/Normal-9010.png,True,False,299,299,L,175.26,9970ce038869cbed75784e8c56926ed2
Normal-9011.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9011.png,Normal/images/Normal-9011.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9011.png,Normal/masks/Normal-9011.png,True,False,299,299,L,160.45,91e771165095a9b3ca6a84b17632c472
Normal-9012.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9012.png,Normal/images/Normal-9012.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9012.png,Normal/masks/Normal-9012.png,True,False,299,299,L,155.96,0afa714e6ffafe14f79519d5f37cba2d
Normal-9013.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9013.png,Normal/images/Normal-9013.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9013.png,Normal/masks/Normal-9013.png,True,False,299,299,L,147.71,c547c019103000217abb5f40dcce4a40
Normal-9014.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9014.png,Normal/images/Normal-9014.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9014.png,Normal/masks/Normal-9014.png,True,False,299,299,L,103.04,706069f6d716522e964289afdc4509da
Normal-9015.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9015.png,Normal/images/Normal-9015.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9015.png,Normal/masks/Normal-9015.png,True,False,299,299,L,184.23,08d48e0142d3ac74590a705100ec3606
Normal-9016.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9016.png,Normal/images/Normal-9016.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9016.png,Normal/masks/Normal-9016.png,True,False,299,299,L,166.7,3916c2cc32b17c77bc99ecd95bd00353
Normal-9017.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9017.png,Normal/images/Normal-9017.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9017.png,Normal/masks/Normal-9017.png,True,False,299,299,L,176.41,2fd81f77aa98f9e7ef0e008873821f5b
Normal-9018.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9018.png,Normal/images/Normal-9018.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9018.png,Normal/masks/Normal-9018.png,True,False,299,299,L,118.97,7eee9074ff6259a1fdd4999cfc993a97
Normal-9019.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9019.png,Normal/images/Normal-9019.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9019.png,Normal/masks/Normal-9019.png,True,False,299,299,L,141.84,2b2aeca50185fb64d9eabb7c2d9ad35d
Normal-902.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-902.png,Normal/images/Normal-902.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-902.png,Normal/masks/Normal-902.png,True,False,299,299,L,140.23,225aa6607577c850ff7a47866bc9d73d
Normal-9020.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9020.png,Normal/images/Normal-9020.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9020.png,Normal/masks/Normal-9020.png,True,False,299,299,L,102.69,18e44ee714061118b6a27dcf1b9f07a6
Normal-9021.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9021.png,Normal/images/Normal-9021.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9021.png,Normal/masks/Normal-9021.png,True,False,299,299,L,126.56,66489f8fe71776fbcc3ad15f0dfc1a9a
Normal-9022.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9022.png,Normal/images/Normal-9022.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9022.png,Normal/masks/Normal-9022.png,True,False,299,299,L,131.53,022871c73eb35484e6297f0be93555c4
Normal-9023.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9023.png,Normal/images/Normal-9023.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9023.png,Normal/masks/Normal-9023.png,True,False,299,299,L,130.77,4f816f3162d50eaa9561a42dd087d802
Normal-9024.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9024.png,Normal/images/Normal-9024.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9024.png,Normal/masks/Normal-9024.png,True,False,299,299,L,117.26,f5c79256ed7eb62d6a27b0b3f2f75281
Normal-9025.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9025.png,Normal/images/Normal-9025.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9025.png,Normal/masks/Normal-9025.png,True,False,299,299,L,122.76,0c9a4bf7c48f83d3322c0a1e8c16e3bc
Normal-9026.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9026.png,Normal/images/Normal-9026.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9026.png,Normal/masks/Normal-9026.png,True,False,299,299,L,166.0,ab93298a6a94e48a3763b07024851701
Normal-9027.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9027.png,Normal/images/Normal-9027.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9027.png,Normal/masks/Normal-9027.png,True,False,299,299,L,123.25,077f533fd89440162a5a6f1920240d56
Normal-9028.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9028.png,Normal/images/Normal-9028.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9028.png,Normal/masks/Normal-9028.png,True,False,299,299,L,158.13,87f57081c33f3efd712b69bc02fd5eb5
Normal-9029.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9029.png,Normal/images/Normal-9029.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9029.png,Normal/masks/Normal-9029.png,True,False,299,299,L,137.89,e5a42321e3b1dbffb98d4d60aaf92d43
Normal-903.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-903.png,Normal/images/Normal-903.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-903.png,Normal/masks/Normal-903.png,True,False,299,299,L,124.83,9b0e2b7235b06c4c2f15bdbeb8b3ef2d
Normal-9030.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9030.png,Normal/images/Normal-9030.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9030.png,Normal/masks/Normal-9030.png,True,False,299,299,L,114.24,c10a5dcb04c793895a1bcd3f9d2983bd
Normal-9031.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9031.png,Normal/images/Normal-9031.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9031.png,Normal/masks/Normal-9031.png,True,False,299,299,L,121.4,8fa19edcf895234faad203c0d4660dd8
Normal-9032.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9032.png,Normal/images/Normal-9032.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9032.png,Normal/masks/Normal-9032.png,True,False,299,299,L,134.98,7c4a0be887427054e788907a7d58f920
Normal-9033.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9033.png,Normal/images/Normal-9033.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9033.png,Normal/masks/Normal-9033.png,True,False,299,299,L,141.57,cd46f7f8dec6a53afcc5a46279769cbe
Normal-9034.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9034.png,Normal/images/Normal-9034.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9034.png,Normal/masks/Normal-9034.png,True,False,299,299,L,110.63,7271329feb7f5a9eb3b8b9cb11760bc5
Normal-9035.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9035.png,Normal/images/Normal-9035.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9035.png,Normal/masks/Normal-9035.png,True,False,299,299,L,134.87,c787ce792a3360efd8ee245ab4517ddf
Normal-9036.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9036.png,Normal/images/Normal-9036.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9036.png,Normal/masks/Normal-9036.png,True,False,299,299,L,116.92,bcd83c17a3f60d4c40a8ee3908bf1492
Normal-9037.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9037.png,Normal/images/Normal-9037.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9037.png,Normal/masks/Normal-9037.png,True,False,299,299,L,157.18,748802800277f04cfe0989f66601dfe6
Normal-9038.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9038.png,Normal/images/Normal-9038.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9038.png,Normal/masks/Normal-9038.png,True,False,299,299,L,135.25,e161fb09111c14e4f18354606cbbbc2e
Normal-9039.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9039.png,Normal/images/Normal-9039.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9039.png,Normal/masks/Normal-9039.png,True,False,299,299,L,133.95,5707d106b8716096b8832493ae29c21a
Normal-904.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-904.png,Normal/images/Normal-904.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-904.png,Normal/masks/Normal-904.png,True,False,299,299,L,114.22,0c8f85ee5044565015a6f404dd5bb1d2
Normal-9040.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9040.png,Normal/images/Normal-9040.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9040.png,Normal/masks/Normal-9040.png,True,False,299,299,L,124.98,bb4b4e08137b2bef7df852ca6c92a2b3
Normal-9041.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9041.png,Normal/images/Normal-9041.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9041.png,Normal/masks/Normal-9041.png,True,False,299,299,L,165.11,2cee10429fdb1bcd0c8134f8396f57e5
Normal-9042.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9042.png,Normal/images/Normal-9042.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9042.png,Normal/masks/Normal-9042.png,True,False,299,299,L,110.81,d5fbc1ae49b772cb89cbeb838c4420ca
Normal-9043.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9043.png,Normal/images/Normal-9043.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9043.png,Normal/masks/Normal-9043.png,True,False,299,299,L,122.24,2fcc487698cd5b326236654a219e8f69
Normal-9044.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9044.png,Normal/images/Normal-9044.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9044.png,Normal/masks/Normal-9044.png,True,False,299,299,L,121.25,678930486b1bb1688124440b45df29d8
Normal-9045.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9045.png,Normal/images/Normal-9045.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9045.png,Normal/masks/Normal-9045.png,True,False,299,299,L,102.15,917e7917c4edb89b4428c5adff900118
Normal-9046.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9046.png,Normal/images/Normal-9046.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9046.png,Normal/masks/Normal-9046.png,True,False,299,299,L,162.64,228a299dc3a33b4f2414470909b34d77
Normal-9047.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9047.png,Normal/images/Normal-9047.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9047.png,Normal/masks/Normal-9047.png,True,False,299,299,L,147.89,0ec8f7c91e62267f0c308b579e3a0c14
Normal-9048.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9048.png,Normal/images/Normal-9048.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9048.png,Normal/masks/Normal-9048.png,True,False,299,299,L,122.48,52dbbf7295f333308b92f19cc7276a6f
Normal-9049.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9049.png,Normal/images/Normal-9049.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9049.png,Normal/masks/Normal-9049.png,True,False,299,299,L,133.03,152010e074036f36829ec4383509ffe8
Normal-905.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-905.png,Normal/images/Normal-905.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-905.png,Normal/masks/Normal-905.png,True,False,299,299,L,149.65,9cccdf58b32147e14410dff98ea781dc
Normal-9050.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9050.png,Normal/images/Normal-9050.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9050.png,Normal/masks/Normal-9050.png,True,False,299,299,L,129.83,1488b653bf9a48c4806df59d33cb1ffc
Normal-9051.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9051.png,Normal/images/Normal-9051.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9051.png,Normal/masks/Normal-9051.png,True,False,299,299,L,166.3,b0cee0cd3e0958c6230bc2c17c503195
Normal-9052.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9052.png,Normal/images/Normal-9052.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9052.png,Normal/masks/Normal-9052.png,True,False,299,299,L,115.61,da188f004bdbc2629f1a4a613e8afe97
Normal-9053.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9053.png,Normal/images/Normal-9053.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9053.png,Normal/masks/Normal-9053.png,True,False,299,299,L,131.77,c0f5efd5ea4d027e16f36e2c1a88554e
Normal-9054.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9054.png,Normal/images/Normal-9054.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9054.png,Normal/masks/Normal-9054.png,True,False,299,299,L,173.93,cb9817ab1e7442e8c2fb37e88baa74d8
Normal-9055.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9055.png,Normal/images/Normal-9055.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9055.png,Normal/masks/Normal-9055.png,True,False,299,299,L,111.01,d3271698af1b02fa18685650bba21fad
Normal-9056.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9056.png,Normal/images/Normal-9056.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9056.png,Normal/masks/Normal-9056.png,True,False,299,299,L,126.84,fd0dd2c899565b8fbbba337c1959c55e
Normal-9057.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9057.png,Normal/images/Normal-9057.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9057.png,Normal/masks/Normal-9057.png,True,False,299,299,L,92.37,82c2deded29a56d87d001ceaa712274c
Normal-9058.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9058.png,Normal/images/Normal-9058.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9058.png,Normal/masks/Normal-9058.png,True,False,299,299,L,127.54,b090e0f39d0e19ec8b5bcaab10135eba
Normal-9059.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9059.png,Normal/images/Normal-9059.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9059.png,Normal/masks/Normal-9059.png,True,False,299,299,L,113.92,9e45f778f84cadd68dc52429db24c2bb
Normal-906.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-906.png,Normal/images/Normal-906.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-906.png,Normal/masks/Normal-906.png,True,False,299,299,L,148.06,954f914721b62fe7487550806789fa72
Normal-9060.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9060.png,Normal/images/Normal-9060.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9060.png,Normal/masks/Normal-9060.png,True,False,299,299,L,133.45,69e36c66f81f817f65f89d60994a277c
Normal-9061.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9061.png,Normal/images/Normal-9061.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9061.png,Normal/masks/Normal-9061.png,True,False,299,299,L,163.25,b0d637fa66f7d169ae03b37a1a12757e
Normal-9062.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9062.png,Normal/images/Normal-9062.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9062.png,Normal/masks/Normal-9062.png,True,False,299,299,L,102.58,dc3cf0a126d5a2ae310f45db18d8c1f0
Normal-9063.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9063.png,Normal/images/Normal-9063.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9063.png,Normal/masks/Normal-9063.png,True,False,299,299,L,132.81,aeb63b537426fef833c70703c3741eda
Normal-9064.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9064.png,Normal/images/Normal-9064.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9064.png,Normal/masks/Normal-9064.png,True,False,299,299,L,139.07,d23f3c6d0bef09be27ab848724cd9cbc
Normal-9065.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9065.png,Normal/images/Normal-9065.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9065.png,Normal/masks/Normal-9065.png,True,False,299,299,L,149.84,d32c3bdebac79e0b70e119ab9839090e
Normal-9066.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9066.png,Normal/images/Normal-9066.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9066.png,Normal/masks/Normal-9066.png,True,False,299,299,L,116.64,0a7e617bbc4a1946f3a5d813a350fd5f
Normal-9067.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9067.png,Normal/images/Normal-9067.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9067.png,Normal/masks/Normal-9067.png,True,False,299,299,L,119.19,f399d94e54f37cccd374acfe3dfbc4fd
Normal-9068.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9068.png,Normal/images/Normal-9068.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9068.png,Normal/masks/Normal-9068.png,True,False,299,299,L,160.8,bdda834e5422487ab1e28025e3d7e739
Normal-9069.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9069.png,Normal/images/Normal-9069.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9069.png,Normal/masks/Normal-9069.png,True,False,299,299,L,165.68,4a98777000fdb4d82f8486f517a32323
Normal-907.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-907.png,Normal/images/Normal-907.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-907.png,Normal/masks/Normal-907.png,True,False,299,299,L,138.5,5d3181c95e27180331d70a369f78ebc8
Normal-9070.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9070.png,Normal/images/Normal-9070.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9070.png,Normal/masks/Normal-9070.png,True,False,299,299,L,106.53,2cf45e143cbc7a53da6bb4c0d697f390
Normal-9071.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9071.png,Normal/images/Normal-9071.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9071.png,Normal/masks/Normal-9071.png,True,False,299,299,L,100.59,ced25f0992560845b47de81f7608e8ac
Normal-9072.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9072.png,Normal/images/Normal-9072.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9072.png,Normal/masks/Normal-9072.png,True,False,299,299,L,124.46,3c0003bcec38d419eb95035c6317e008
Normal-9073.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9073.png,Normal/images/Normal-9073.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9073.png,Normal/masks/Normal-9073.png,True,False,299,299,L,129.53,232e248be59154cecf3d599c51bb2491
Normal-9074.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9074.png,Normal/images/Normal-9074.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9074.png,Normal/masks/Normal-9074.png,True,False,299,299,L,113.1,5a029ba7483bad0fd71274d3a76d2f9d
Normal-9075.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9075.png,Normal/images/Normal-9075.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9075.png,Normal/masks/Normal-9075.png,True,False,299,299,L,102.97,916d1c0f939bb377ee601373e9629239
Normal-9076.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9076.png,Normal/images/Normal-9076.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9076.png,Normal/masks/Normal-9076.png,True,False,299,299,L,163.47,2a557f3e604ba78b44fd1a5773da85cb
Normal-9077.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9077.png,Normal/images/Normal-9077.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9077.png,Normal/masks/Normal-9077.png,True,False,299,299,L,123.65,ee9e6ab61109fc830b233eb522b18b96
Normal-9078.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9078.png,Normal/images/Normal-9078.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9078.png,Normal/masks/Normal-9078.png,True,False,299,299,L,124.24,b6d12eda353bc328071534c084f71626
Normal-9079.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9079.png,Normal/images/Normal-9079.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9079.png,Normal/masks/Normal-9079.png,True,False,299,299,L,144.02,ed9df79419876a1f8a45913e1cbe4c70
Normal-908.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-908.png,Normal/images/Normal-908.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-908.png,Normal/masks/Normal-908.png,True,False,299,299,L,127.45,6dd3ab9a208fa892bb5e7ccb5d0d669b
Normal-9080.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9080.png,Normal/images/Normal-9080.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9080.png,Normal/masks/Normal-9080.png,True,False,299,299,L,121.83,c1defa8f58f27477e8545788f2d3bcde
Normal-9081.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9081.png,Normal/images/Normal-9081.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9081.png,Normal/masks/Normal-9081.png,True,False,299,299,L,164.7,bbafc1ba1c545ccfef4d8517bd7e39ae
Normal-9082.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9082.png,Normal/images/Normal-9082.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9082.png,Normal/masks/Normal-9082.png,True,False,299,299,L,142.47,d20690c467b726d78c011129a9e8c708
Normal-9083.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9083.png,Normal/images/Normal-9083.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9083.png,Normal/masks/Normal-9083.png,True,False,299,299,L,108.99,1f33fd03b7ff1d84e49ae68b77345c53
Normal-9084.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9084.png,Normal/images/Normal-9084.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9084.png,Normal/masks/Normal-9084.png,True,False,299,299,L,130.56,7dcc74d32cc298626fb1c09bd0786cd4
Normal-9085.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9085.png,Normal/images/Normal-9085.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9085.png,Normal/masks/Normal-9085.png,True,False,299,299,L,95.15,e255c4c098904f5c2cbe7a419f49e5da
Normal-9086.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9086.png,Normal/images/Normal-9086.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9086.png,Normal/masks/Normal-9086.png,True,False,299,299,L,155.88,0e41df16efac937d09d15d24c1f3b33a
Normal-9087.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9087.png,Normal/images/Normal-9087.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9087.png,Normal/masks/Normal-9087.png,True,False,299,299,L,155.78,99f24b16b4523088984a51d547ec85c9
Normal-9088.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9088.png,Normal/images/Normal-9088.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9088.png,Normal/masks/Normal-9088.png,True,False,299,299,L,101.28,064ff632585202765a446cdc5a4d93d8
Normal-9089.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9089.png,Normal/images/Normal-9089.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9089.png,Normal/masks/Normal-9089.png,True,False,299,299,L,140.44,4f479c93b33a4a4a49571532acc3190a
Normal-909.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-909.png,Normal/images/Normal-909.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-909.png,Normal/masks/Normal-909.png,True,False,299,299,L,119.11,a0ecb07f053db428fb12b7c696e6a4e4
Normal-9090.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9090.png,Normal/images/Normal-9090.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9090.png,Normal/masks/Normal-9090.png,True,False,299,299,L,122.89,d1007a86b140d314c2808568cfd3d8df
Normal-9091.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9091.png,Normal/images/Normal-9091.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9091.png,Normal/masks/Normal-9091.png,True,False,299,299,L,157.75,5a5b35b2ff460c782f8c97f7a3762f39
Normal-9092.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9092.png,Normal/images/Normal-9092.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9092.png,Normal/masks/Normal-9092.png,True,False,299,299,L,161.72,f8e86855a702eae2925fa455c098a2e2
Normal-9093.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9093.png,Normal/images/Normal-9093.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9093.png,Normal/masks/Normal-9093.png,True,False,299,299,L,102.56,349137550e76af922ce227205ef79ce0
Normal-9094.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9094.png,Normal/images/Normal-9094.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9094.png,Normal/masks/Normal-9094.png,True,False,299,299,L,166.3,56cdb01f0d90a1e65a0bd602ffe5ce8c
Normal-9095.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9095.png,Normal/images/Normal-9095.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9095.png,Normal/masks/Normal-9095.png,True,False,299,299,L,119.31,fe7a35f68e0ac9e2c89b1686a22797f7
Normal-9096.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9096.png,Normal/images/Normal-9096.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9096.png,Normal/masks/Normal-9096.png,True,False,299,299,L,117.48,920fa6a58d0b7874ea7ae63cb5f711a5
Normal-9097.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9097.png,Normal/images/Normal-9097.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9097.png,Normal/masks/Normal-9097.png,True,False,299,299,L,137.82,0cadca1b822df12da8b1fdebd152dc80
Normal-9098.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9098.png,Normal/images/Normal-9098.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9098.png,Normal/masks/Normal-9098.png,True,False,299,299,L,116.01,7810c6458bf76f1ea9ad3bcaafed0fc9
Normal-9099.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9099.png,Normal/images/Normal-9099.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9099.png,Normal/masks/Normal-9099.png,True,False,299,299,L,116.16,c45fb5e4650dbbe0c51ed9c15f0aa86b
Normal-91.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-91.png,Normal/images/Normal-91.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-91.png,Normal/masks/Normal-91.png,True,False,299,299,L,148.97,750df3465caeca828193cbf3dcf5a94a
Normal-910.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-910.png,Normal/images/Normal-910.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-910.png,Normal/masks/Normal-910.png,True,False,299,299,L,140.95,ac43d1a2d18de0f8be5ffdb631826dca
Normal-9100.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9100.png,Normal/images/Normal-9100.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9100.png,Normal/masks/Normal-9100.png,True,False,299,299,L,135.56,b5b1df3fa7ad7c2a097b8e5bcd87e7fb
Normal-9101.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9101.png,Normal/images/Normal-9101.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9101.png,Normal/masks/Normal-9101.png,True,False,299,299,L,92.66,050d710727659a3576b5f1dfc67a9b4a
Normal-9102.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9102.png,Normal/images/Normal-9102.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9102.png,Normal/masks/Normal-9102.png,True,False,299,299,L,126.6,e1109a77d47ca99cfa2fd4fdf3d6dcd8
Normal-9103.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9103.png,Normal/images/Normal-9103.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9103.png,Normal/masks/Normal-9103.png,True,False,299,299,L,106.94,923ba82e98fbf52730dcc0e37a110b65
Normal-9104.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9104.png,Normal/images/Normal-9104.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9104.png,Normal/masks/Normal-9104.png,True,False,299,299,L,121.62,ead5b29823c792a9fd4c304b7965c7cc
Normal-9105.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9105.png,Normal/images/Normal-9105.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9105.png,Normal/masks/Normal-9105.png,True,False,299,299,L,99.09,cdaad875ac5cd4421e9a72a85398e369
Normal-9106.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9106.png,Normal/images/Normal-9106.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9106.png,Normal/masks/Normal-9106.png,True,False,299,299,L,147.4,ad2effc88aab880dfb10beaad44fcdde
Normal-9107.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9107.png,Normal/images/Normal-9107.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9107.png,Normal/masks/Normal-9107.png,True,False,299,299,L,127.2,1a826e97ab91574311bbb67ef80c09ee
Normal-9108.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9108.png,Normal/images/Normal-9108.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9108.png,Normal/masks/Normal-9108.png,True,False,299,299,L,118.15,887dc854f7b137574c69c81bdd5456d9
Normal-9109.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9109.png,Normal/images/Normal-9109.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9109.png,Normal/masks/Normal-9109.png,True,False,299,299,L,107.24,bd1a975d34495b57ecf4d1517832e524
Normal-911.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-911.png,Normal/images/Normal-911.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-911.png,Normal/masks/Normal-911.png,True,False,299,299,L,120.12,4777d10f761f1939ef30c2c1f0d87c44
Normal-9110.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9110.png,Normal/images/Normal-9110.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9110.png,Normal/masks/Normal-9110.png,True,False,299,299,L,147.52,82504435db3f2b1ca7a07152ca7ee10b
Normal-9111.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9111.png,Normal/images/Normal-9111.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9111.png,Normal/masks/Normal-9111.png,True,False,299,299,L,174.73,b4aacf4d7769296c787eef79b56d03dd
Normal-9112.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9112.png,Normal/images/Normal-9112.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9112.png,Normal/masks/Normal-9112.png,True,False,299,299,L,98.86,bfb9b292a901463c666dd09f6a1c4064
Normal-9113.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9113.png,Normal/images/Normal-9113.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9113.png,Normal/masks/Normal-9113.png,True,False,299,299,L,128.52,c38818bd6ee208c1d63cac71d2aca37c
Normal-9114.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9114.png,Normal/images/Normal-9114.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9114.png,Normal/masks/Normal-9114.png,True,False,299,299,L,150.26,2bcade86a4efbe6c98b27317b0b8a3ea
Normal-9115.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9115.png,Normal/images/Normal-9115.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9115.png,Normal/masks/Normal-9115.png,True,False,299,299,L,90.07,a968c4a778925d8dfaea42e15f41c0e2
Normal-9116.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9116.png,Normal/images/Normal-9116.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9116.png,Normal/masks/Normal-9116.png,True,False,299,299,L,106.22,d00a8c355fe0666b79d707f141c5ff98
Normal-9117.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9117.png,Normal/images/Normal-9117.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9117.png,Normal/masks/Normal-9117.png,True,False,299,299,L,61.7,d4f3c002e64a5ed67889668e23818e37
Normal-9118.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9118.png,Normal/images/Normal-9118.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9118.png,Normal/masks/Normal-9118.png,True,False,299,299,L,148.53,027d5c4d4b91086f807f8b200779e8a4
Normal-9119.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9119.png,Normal/images/Normal-9119.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9119.png,Normal/masks/Normal-9119.png,True,False,299,299,L,140.59,549f2f746dc49d1a2ff310f4d981c8f3
Normal-912.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-912.png,Normal/images/Normal-912.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-912.png,Normal/masks/Normal-912.png,True,False,299,299,L,104.46,2efd351a694e2985ae3f21f6897b2125
Normal-9120.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9120.png,Normal/images/Normal-9120.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9120.png,Normal/masks/Normal-9120.png,True,False,299,299,L,138.25,2f0eb733cf7d1ed530f6f21c3c96e2b0
Normal-9121.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9121.png,Normal/images/Normal-9121.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9121.png,Normal/masks/Normal-9121.png,True,False,299,299,L,115.78,23e9d4a14ba0d20ff8c845757f14a2cd
Normal-9122.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9122.png,Normal/images/Normal-9122.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9122.png,Normal/masks/Normal-9122.png,True,False,299,299,L,136.91,c2a602b984f7c744c513efa7deb9a837
Normal-9123.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9123.png,Normal/images/Normal-9123.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9123.png,Normal/masks/Normal-9123.png,True,False,299,299,L,167.82,cd9e6120c003d98cccc3605bc17feb8f
Normal-9124.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9124.png,Normal/images/Normal-9124.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9124.png,Normal/masks/Normal-9124.png,True,False,299,299,L,108.45,4644918d18fe24b1caf42a902e3a94f7
Normal-9125.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9125.png,Normal/images/Normal-9125.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9125.png,Normal/masks/Normal-9125.png,True,False,299,299,L,103.94,cd9baeddbc4a30926ed4d2c4613fd284
Normal-9126.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9126.png,Normal/images/Normal-9126.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9126.png,Normal/masks/Normal-9126.png,True,False,299,299,L,128.49,7d27c85b52d7ed400c6314d31899eb88
Normal-9127.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9127.png,Normal/images/Normal-9127.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9127.png,Normal/masks/Normal-9127.png,True,False,299,299,L,125.9,853f5725cb1f79a6808eeb1623bb731f
Normal-9128.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9128.png,Normal/images/Normal-9128.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9128.png,Normal/masks/Normal-9128.png,True,False,299,299,L,106.08,48aa4cf777429617b779ebfa02ae9582
Normal-9129.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9129.png,Normal/images/Normal-9129.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9129.png,Normal/masks/Normal-9129.png,True,False,299,299,L,157.97,6295e11f89db44d2717b43f6634dc8a3
Normal-913.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-913.png,Normal/images/Normal-913.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-913.png,Normal/masks/Normal-913.png,True,False,299,299,L,124.3,8316b0084aedc46f8fc2ad425e4e6084
Normal-9130.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9130.png,Normal/images/Normal-9130.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9130.png,Normal/masks/Normal-9130.png,True,False,299,299,L,156.7,fa60089f2998f088b44291f21d5468a5
Normal-9131.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9131.png,Normal/images/Normal-9131.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9131.png,Normal/masks/Normal-9131.png,True,False,299,299,L,154.75,f6fdfd3a3e19001f20986104e5a52585
Normal-9132.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9132.png,Normal/images/Normal-9132.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9132.png,Normal/masks/Normal-9132.png,True,False,299,299,L,121.57,0213638e984c2f351e7752bafb8b1185
Normal-9133.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9133.png,Normal/images/Normal-9133.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9133.png,Normal/masks/Normal-9133.png,True,False,299,299,L,157.1,5dfffef6a610837e924c5e5cd8e26bb9
Normal-9134.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9134.png,Normal/images/Normal-9134.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9134.png,Normal/masks/Normal-9134.png,True,False,299,299,L,132.38,0e35d5a395ad3ddeeae7db408a2a5c43
Normal-9135.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9135.png,Normal/images/Normal-9135.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9135.png,Normal/masks/Normal-9135.png,True,False,299,299,L,142.53,0852a05dd55136a9212460d83affe6f1
Normal-9136.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9136.png,Normal/images/Normal-9136.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9136.png,Normal/masks/Normal-9136.png,True,False,299,299,L,123.75,da92848aa70876a51dbcbc84f5cf05e5
Normal-9137.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9137.png,Normal/images/Normal-9137.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9137.png,Normal/masks/Normal-9137.png,True,False,299,299,L,119.93,0fd5ceb1d2748882aa52325c3fc51244
Normal-9138.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9138.png,Normal/images/Normal-9138.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9138.png,Normal/masks/Normal-9138.png,True,False,299,299,L,138.12,4c8fb3b8ad7461383aee90b2d381208a
Normal-9139.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9139.png,Normal/images/Normal-9139.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9139.png,Normal/masks/Normal-9139.png,True,False,299,299,L,108.82,5c9c49b42ca95e9585758e99fd321bc3
Normal-914.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-914.png,Normal/images/Normal-914.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-914.png,Normal/masks/Normal-914.png,True,False,299,299,L,116.95,10f4a7786b4577d3c96f17e694976398
Normal-9140.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9140.png,Normal/images/Normal-9140.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9140.png,Normal/masks/Normal-9140.png,True,False,299,299,L,175.47,11a51453f98214fbfde2a9c0ce6e6c29
Normal-9141.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9141.png,Normal/images/Normal-9141.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9141.png,Normal/masks/Normal-9141.png,True,False,299,299,L,134.63,e62d4ce7b025fecb011f3470175dac79
Normal-9142.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9142.png,Normal/images/Normal-9142.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9142.png,Normal/masks/Normal-9142.png,True,False,299,299,L,118.32,101894054862f2bb4878c8bce4ab2014
Normal-9143.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9143.png,Normal/images/Normal-9143.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9143.png,Normal/masks/Normal-9143.png,True,False,299,299,L,127.28,a8ad1e6c50a1295cbfc6d93be61b6ba9
Normal-9144.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9144.png,Normal/images/Normal-9144.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9144.png,Normal/masks/Normal-9144.png,True,False,299,299,L,177.03,a2ace9debf1fda9604557b8f539d96f8
Normal-9145.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9145.png,Normal/images/Normal-9145.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9145.png,Normal/masks/Normal-9145.png,True,False,299,299,L,98.93,a704fbe8cf9ed5659fa5d1ffc6c57856
Normal-9146.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9146.png,Normal/images/Normal-9146.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9146.png,Normal/masks/Normal-9146.png,True,False,299,299,L,168.31,6973d32ce2ac84e69ddadebe18da3b87
Normal-9147.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9147.png,Normal/images/Normal-9147.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9147.png,Normal/masks/Normal-9147.png,True,False,299,299,L,136.88,46eb97a8b96b18509788a383fe10ec84
Normal-9148.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9148.png,Normal/images/Normal-9148.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9148.png,Normal/masks/Normal-9148.png,True,False,299,299,L,89.52,aab6b88d9afaba2ff2035ca1b1999be9
Normal-9149.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9149.png,Normal/images/Normal-9149.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9149.png,Normal/masks/Normal-9149.png,True,False,299,299,L,106.64,f2da9c171e8c4c4b18c0ebaa5ccd811d
Normal-915.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-915.png,Normal/images/Normal-915.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-915.png,Normal/masks/Normal-915.png,True,False,299,299,L,73.3,565c1da29fbfe563f4e392909ed0800f
Normal-9150.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9150.png,Normal/images/Normal-9150.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9150.png,Normal/masks/Normal-9150.png,True,False,299,299,L,163.17,9ccdc223a1057e8bcc19f0ab1199aef2
Normal-9151.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9151.png,Normal/images/Normal-9151.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9151.png,Normal/masks/Normal-9151.png,True,False,299,299,L,108.01,e6c729d88538be57adcaf44214d914b8
Normal-9152.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9152.png,Normal/images/Normal-9152.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9152.png,Normal/masks/Normal-9152.png,True,False,299,299,L,77.17,f1193a190dd7eabf3f7105f691d1668e
Normal-9153.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9153.png,Normal/images/Normal-9153.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9153.png,Normal/masks/Normal-9153.png,True,False,299,299,L,117.72,d7d1e8d859fa4501e32a8bc0f5dca92d
Normal-9154.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9154.png,Normal/images/Normal-9154.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9154.png,Normal/masks/Normal-9154.png,True,False,299,299,L,159.35,f67004628e0113b2341b7af364850fd8
Normal-9155.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9155.png,Normal/images/Normal-9155.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9155.png,Normal/masks/Normal-9155.png,True,False,299,299,L,121.21,35a9cbf1e439fb6ea95505f976bb6a92
Normal-9156.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9156.png,Normal/images/Normal-9156.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9156.png,Normal/masks/Normal-9156.png,True,False,299,299,L,132.68,f6de89b69b59620e25ae1035c1e43426
Normal-9157.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9157.png,Normal/images/Normal-9157.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9157.png,Normal/masks/Normal-9157.png,True,False,299,299,L,183.85,e6e5bc841f9ecbc2243874f19f9a7708
Normal-9158.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9158.png,Normal/images/Normal-9158.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9158.png,Normal/masks/Normal-9158.png,True,False,299,299,L,122.36,bd1b3af2ef60e1228c40c2690b8576e6
Normal-9159.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9159.png,Normal/images/Normal-9159.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9159.png,Normal/masks/Normal-9159.png,True,False,299,299,L,102.08,0ba843ce51f1883dc30d0f602a8d7eab
Normal-916.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-916.png,Normal/images/Normal-916.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-916.png,Normal/masks/Normal-916.png,True,False,299,299,L,102.78,a2043956e2570a505766b2a47253443d
Normal-9160.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9160.png,Normal/images/Normal-9160.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9160.png,Normal/masks/Normal-9160.png,True,False,299,299,L,80.58,9ffceefcd72c4a17c9bf35ad2afada6c
Normal-9161.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9161.png,Normal/images/Normal-9161.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9161.png,Normal/masks/Normal-9161.png,True,False,299,299,L,157.44,1df2dfc7f7166f5f7ddf197ea88e1c28
Normal-9162.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9162.png,Normal/images/Normal-9162.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9162.png,Normal/masks/Normal-9162.png,True,False,299,299,L,123.02,085ebc3f9206a7a7c3f7fba216a53dd8
Normal-9163.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9163.png,Normal/images/Normal-9163.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9163.png,Normal/masks/Normal-9163.png,True,False,299,299,L,115.12,739d79a561408f79a72f72d60b97372e
Normal-9164.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9164.png,Normal/images/Normal-9164.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9164.png,Normal/masks/Normal-9164.png,True,False,299,299,L,116.99,d42645a8912d1571607876502598ea24
Normal-9165.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9165.png,Normal/images/Normal-9165.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9165.png,Normal/masks/Normal-9165.png,True,False,299,299,L,115.92,92b021b2ffa66101742d3f06428bb39f
Normal-9166.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9166.png,Normal/images/Normal-9166.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9166.png,Normal/masks/Normal-9166.png,True,False,299,299,L,121.98,49d0d130aa59c103eb166da9479ba355
Normal-9167.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9167.png,Normal/images/Normal-9167.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9167.png,Normal/masks/Normal-9167.png,True,False,299,299,L,112.98,d95e13d83548d1cb09a6d36f3a914b81
Normal-9168.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9168.png,Normal/images/Normal-9168.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9168.png,Normal/masks/Normal-9168.png,True,False,299,299,L,154.04,753d0281ef6c58e2d4eb7de928638d22
Normal-9169.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9169.png,Normal/images/Normal-9169.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9169.png,Normal/masks/Normal-9169.png,True,False,299,299,L,135.3,b0ad31de0f34c951ce94ebc8cc062a47
Normal-917.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-917.png,Normal/images/Normal-917.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-917.png,Normal/masks/Normal-917.png,True,False,299,299,L,151.93,815d54b3be16eb46415858256a88d7c8
Normal-9170.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9170.png,Normal/images/Normal-9170.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9170.png,Normal/masks/Normal-9170.png,True,False,299,299,L,127.93,2b42c4fde7c14013b81b648984d2c7ff
Normal-9171.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9171.png,Normal/images/Normal-9171.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9171.png,Normal/masks/Normal-9171.png,True,False,299,299,L,129.74,30e1fea616beb536c1aa6ebdaa042ae8
Normal-9172.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9172.png,Normal/images/Normal-9172.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9172.png,Normal/masks/Normal-9172.png,True,False,299,299,L,114.78,03cb3cdefc02a20221a7cffe69ff06cc
Normal-9173.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9173.png,Normal/images/Normal-9173.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9173.png,Normal/masks/Normal-9173.png,True,False,299,299,L,154.81,ff0804d868ed65d5320e95d95d69ca45
Normal-9174.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9174.png,Normal/images/Normal-9174.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9174.png,Normal/masks/Normal-9174.png,True,False,299,299,L,149.53,c2ba091b04921a7a01a47c3e852b1873
Normal-9175.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9175.png,Normal/images/Normal-9175.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9175.png,Normal/masks/Normal-9175.png,True,False,299,299,L,163.59,3bcd57d029848500918f6855ad2ffcaf
Normal-9176.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9176.png,Normal/images/Normal-9176.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9176.png,Normal/masks/Normal-9176.png,True,False,299,299,L,124.24,9923ba21287ea424c108b60fad9a1283
Normal-9177.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9177.png,Normal/images/Normal-9177.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9177.png,Normal/masks/Normal-9177.png,True,False,299,299,L,167.25,236fbab2bfdd708ccbbd57bd416010bb
Normal-9178.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9178.png,Normal/images/Normal-9178.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9178.png,Normal/masks/Normal-9178.png,True,False,299,299,L,94.91,f64074de4ff83746f1de336063ef84ce
Normal-9179.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9179.png,Normal/images/Normal-9179.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9179.png,Normal/masks/Normal-9179.png,True,False,299,299,L,153.02,571494463c98fd8c016da8be5d9dd843
Normal-918.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-918.png,Normal/images/Normal-918.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-918.png,Normal/masks/Normal-918.png,True,False,299,299,L,114.95,d4029626c2f114ec4909a8115ef9c787
Normal-9180.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9180.png,Normal/images/Normal-9180.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9180.png,Normal/masks/Normal-9180.png,True,False,299,299,L,107.27,534daf9b7150bd2f94bae88f3008f855
Normal-9181.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9181.png,Normal/images/Normal-9181.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9181.png,Normal/masks/Normal-9181.png,True,False,299,299,L,125.3,24cdabfa2568d4ad8e134390be989525
Normal-9182.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9182.png,Normal/images/Normal-9182.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9182.png,Normal/masks/Normal-9182.png,True,False,299,299,L,130.2,218b858f7e1832d77eeb3a48b35d37d1
Normal-9183.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9183.png,Normal/images/Normal-9183.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9183.png,Normal/masks/Normal-9183.png,True,False,299,299,L,120.26,fd8e26ffb0cda277b60490c24b0329e9
Normal-9184.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9184.png,Normal/images/Normal-9184.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9184.png,Normal/masks/Normal-9184.png,True,False,299,299,L,181.48,2ab45e199e99e15751d533ce596f0312
Normal-9185.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9185.png,Normal/images/Normal-9185.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9185.png,Normal/masks/Normal-9185.png,True,False,299,299,L,109.33,dfef49268c1336dd71b678fdb27739b8
Normal-9186.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9186.png,Normal/images/Normal-9186.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9186.png,Normal/masks/Normal-9186.png,True,False,299,299,L,122.13,576f2669d18c14eae1b90b0b327b5572
Normal-9187.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9187.png,Normal/images/Normal-9187.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9187.png,Normal/masks/Normal-9187.png,True,False,299,299,L,103.91,678c46882387611670e039939fe8775f
Normal-9188.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9188.png,Normal/images/Normal-9188.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9188.png,Normal/masks/Normal-9188.png,True,False,299,299,L,124.92,21bc2a671f61452f2d181e79f7e2d12d
Normal-9189.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9189.png,Normal/images/Normal-9189.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9189.png,Normal/masks/Normal-9189.png,True,False,299,299,L,112.53,bc2fb32f07f03eb7b7590d7966e36ae6
Normal-919.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-919.png,Normal/images/Normal-919.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-919.png,Normal/masks/Normal-919.png,True,False,299,299,L,106.98,6bea014939936c85079fab0f6beddad2
Normal-9190.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9190.png,Normal/images/Normal-9190.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9190.png,Normal/masks/Normal-9190.png,True,False,299,299,L,122.69,84cf168e62bfb0a5cba905b7933e1ec6
Normal-9191.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9191.png,Normal/images/Normal-9191.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9191.png,Normal/masks/Normal-9191.png,True,False,299,299,L,109.15,9baadcd6321cc17edf73d8ef19ce9920
Normal-9192.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9192.png,Normal/images/Normal-9192.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9192.png,Normal/masks/Normal-9192.png,True,False,299,299,L,179.67,7f21646ebb013ffbbbb5b076ff0274bf
Normal-9193.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9193.png,Normal/images/Normal-9193.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9193.png,Normal/masks/Normal-9193.png,True,False,299,299,L,133.68,79158d41b7739ca4392d6156c05f8166
Normal-9194.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9194.png,Normal/images/Normal-9194.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9194.png,Normal/masks/Normal-9194.png,True,False,299,299,L,161.27,a1bdc83160a8adbfc063adacb4499ae8
Normal-9195.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9195.png,Normal/images/Normal-9195.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9195.png,Normal/masks/Normal-9195.png,True,False,299,299,L,114.12,9046fbaa6fec5c8dbd9e7ed9cc40a289
Normal-9196.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9196.png,Normal/images/Normal-9196.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9196.png,Normal/masks/Normal-9196.png,True,False,299,299,L,122.77,b4adfec3b2234ac4d208ce46d831d465
Normal-9197.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9197.png,Normal/images/Normal-9197.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9197.png,Normal/masks/Normal-9197.png,True,False,299,299,L,145.0,3b695648a0a87ae4651cc3e0d28e3752
Normal-9198.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9198.png,Normal/images/Normal-9198.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9198.png,Normal/masks/Normal-9198.png,True,False,299,299,L,124.78,8d1fa162dff2ca76621c5ca062601f11
Normal-9199.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9199.png,Normal/images/Normal-9199.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9199.png,Normal/masks/Normal-9199.png,True,False,299,299,L,115.58,1d9b6644390b05974c5c0ba8a4474434
Normal-92.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-92.png,Normal/images/Normal-92.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-92.png,Normal/masks/Normal-92.png,True,False,299,299,L,141.04,547b9333e667dcc4cc8e0e01ecb33246
Normal-920.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-920.png,Normal/images/Normal-920.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-920.png,Normal/masks/Normal-920.png,True,False,299,299,L,121.49,1d498e091c44a8c2b692ad32e610a9c2
Normal-9200.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9200.png,Normal/images/Normal-9200.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9200.png,Normal/masks/Normal-9200.png,True,False,299,299,L,105.5,c0aaa9efa4c8d0be5670f9c8f995f046
Normal-9201.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9201.png,Normal/images/Normal-9201.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9201.png,Normal/masks/Normal-9201.png,True,False,299,299,L,167.09,75471f4caa082097059b9f9d3e5ff220
Normal-9202.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9202.png,Normal/images/Normal-9202.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9202.png,Normal/masks/Normal-9202.png,True,False,299,299,L,139.08,3c33b5251b147c45f69eaf279d369b54
Normal-9203.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9203.png,Normal/images/Normal-9203.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9203.png,Normal/masks/Normal-9203.png,True,False,299,299,L,134.47,580fa82b6307d048a0a59d7860a90697
Normal-9204.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9204.png,Normal/images/Normal-9204.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9204.png,Normal/masks/Normal-9204.png,True,False,299,299,L,158.6,2501944c73b9818a679176175fbbb42b
Normal-9205.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9205.png,Normal/images/Normal-9205.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9205.png,Normal/masks/Normal-9205.png,True,False,299,299,L,175.84,b91d36089c7d34a02e5bf2649ef2c8e3
Normal-9206.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9206.png,Normal/images/Normal-9206.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9206.png,Normal/masks/Normal-9206.png,True,False,299,299,L,136.44,142c2375f2cc4c0cede1ff99fc7d0dd0
Normal-9207.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9207.png,Normal/images/Normal-9207.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9207.png,Normal/masks/Normal-9207.png,True,False,299,299,L,98.23,3265602c311352895aa6645fcc0b8c68
Normal-9208.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9208.png,Normal/images/Normal-9208.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9208.png,Normal/masks/Normal-9208.png,True,False,299,299,L,165.72,6725e07ba444308e512253ef9c9cd4bb
Normal-9209.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9209.png,Normal/images/Normal-9209.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9209.png,Normal/masks/Normal-9209.png,True,False,299,299,L,134.66,eae8bde9654fb1c97320cf6d3f7ffcf7
Normal-921.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-921.png,Normal/images/Normal-921.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-921.png,Normal/masks/Normal-921.png,True,False,299,299,L,151.66,32fffd1a948b48b7988b6a6ec2eee694
Normal-9210.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9210.png,Normal/images/Normal-9210.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9210.png,Normal/masks/Normal-9210.png,True,False,299,299,L,93.43,eea9930a37ddf1f624505f95378dcef2
Normal-9211.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9211.png,Normal/images/Normal-9211.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9211.png,Normal/masks/Normal-9211.png,True,False,299,299,L,119.12,f545ba06bd77a03160eb4e8bf0633c5d
Normal-9212.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9212.png,Normal/images/Normal-9212.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9212.png,Normal/masks/Normal-9212.png,True,False,299,299,L,164.84,45a2a832420d8b7d61ac27a5261ff74d
Normal-9213.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9213.png,Normal/images/Normal-9213.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9213.png,Normal/masks/Normal-9213.png,True,False,299,299,L,122.42,2fa31e695ae6b769f51e017a50b9e8cd
Normal-9214.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9214.png,Normal/images/Normal-9214.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9214.png,Normal/masks/Normal-9214.png,True,False,299,299,L,163.98,bf1aa3a67942b95e51016b00f52fa6c9
Normal-9215.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9215.png,Normal/images/Normal-9215.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9215.png,Normal/masks/Normal-9215.png,True,False,299,299,L,117.37,65e00832be44d8a915feff2e015c5f4e
Normal-9216.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9216.png,Normal/images/Normal-9216.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9216.png,Normal/masks/Normal-9216.png,True,False,299,299,L,129.44,359be2127de572aafb14c6d1c3ced038
Normal-9217.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9217.png,Normal/images/Normal-9217.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9217.png,Normal/masks/Normal-9217.png,True,False,299,299,L,105.98,be97ace5da5fa3fdac73369dc487a3d8
Normal-9218.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9218.png,Normal/images/Normal-9218.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9218.png,Normal/masks/Normal-9218.png,True,False,299,299,L,148.63,183f96b76162c31b9be720ab7cc8f1f3
Normal-9219.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9219.png,Normal/images/Normal-9219.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9219.png,Normal/masks/Normal-9219.png,True,False,299,299,L,136.73,21d33ef548ed971c3c67970287b88231
Normal-922.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-922.png,Normal/images/Normal-922.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-922.png,Normal/masks/Normal-922.png,True,False,299,299,L,130.64,b2463bc815c43d40c8cc569fb698437b
Normal-9220.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9220.png,Normal/images/Normal-9220.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9220.png,Normal/masks/Normal-9220.png,True,False,299,299,L,100.02,d720e8b5572703b59760ee6ea2433bc8
Normal-9221.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9221.png,Normal/images/Normal-9221.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9221.png,Normal/masks/Normal-9221.png,True,False,299,299,L,149.99,a5673939de1efe8d675d2893068dfa9d
Normal-9222.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9222.png,Normal/images/Normal-9222.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9222.png,Normal/masks/Normal-9222.png,True,False,299,299,L,168.23,8ee8dd2acd27ba7719bef10ec2a85993
Normal-9223.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9223.png,Normal/images/Normal-9223.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9223.png,Normal/masks/Normal-9223.png,True,False,299,299,L,137.68,28e7d721339725c3bd43059179fbc957
Normal-9224.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9224.png,Normal/images/Normal-9224.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9224.png,Normal/masks/Normal-9224.png,True,False,299,299,L,114.02,c859447708922565a7cc925768bbbefa
Normal-9225.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9225.png,Normal/images/Normal-9225.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9225.png,Normal/masks/Normal-9225.png,True,False,299,299,L,146.4,4c96f3f4b32a5383252618d23c01eacf
Normal-9226.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9226.png,Normal/images/Normal-9226.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9226.png,Normal/masks/Normal-9226.png,True,False,299,299,L,157.93,df2039ba757aa80c6461c25084fd3a20
Normal-9227.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9227.png,Normal/images/Normal-9227.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9227.png,Normal/masks/Normal-9227.png,True,False,299,299,L,134.76,a81d0e2ac1f6e759b0dd3b35b8262cf1
Normal-9228.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9228.png,Normal/images/Normal-9228.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9228.png,Normal/masks/Normal-9228.png,True,False,299,299,L,96.29,455927baada5caabca70409216ba12f9
Normal-9229.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9229.png,Normal/images/Normal-9229.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9229.png,Normal/masks/Normal-9229.png,True,False,299,299,L,135.98,c2ffec996a8f73e88f84d1690d6b5e8d
Normal-923.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-923.png,Normal/images/Normal-923.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-923.png,Normal/masks/Normal-923.png,True,False,299,299,L,140.58,72ee465fd3457db94bbfb48dd9f9e15c
Normal-9230.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9230.png,Normal/images/Normal-9230.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9230.png,Normal/masks/Normal-9230.png,True,False,299,299,L,131.24,d73d8e3be04c8ea633914d691ba489e8
Normal-9231.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9231.png,Normal/images/Normal-9231.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9231.png,Normal/masks/Normal-9231.png,True,False,299,299,L,117.79,e30de1627194e64773b8c23b2dde64d4
Normal-9232.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9232.png,Normal/images/Normal-9232.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9232.png,Normal/masks/Normal-9232.png,True,False,299,299,L,138.64,a6498d67c4e98c0c3c2a66f8b283b60d
Normal-9233.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9233.png,Normal/images/Normal-9233.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9233.png,Normal/masks/Normal-9233.png,True,False,299,299,L,180.51,6a0ebcbf7c91557744a13dcca0ab4aa8
Normal-9234.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9234.png,Normal/images/Normal-9234.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9234.png,Normal/masks/Normal-9234.png,True,False,299,299,L,101.99,906cfea35434326bb922dd082982fa00
Normal-9235.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9235.png,Normal/images/Normal-9235.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9235.png,Normal/masks/Normal-9235.png,True,False,299,299,L,116.47,a82ce157cfb44c946b64dee0482a21bd
Normal-9236.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9236.png,Normal/images/Normal-9236.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9236.png,Normal/masks/Normal-9236.png,True,False,299,299,L,108.64,cdbf5b3472f1406c659a1ffc31c68c04
Normal-9237.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9237.png,Normal/images/Normal-9237.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9237.png,Normal/masks/Normal-9237.png,True,False,299,299,L,140.7,7a14df23e3abc725e09c64aa8003a823
Normal-9238.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9238.png,Normal/images/Normal-9238.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9238.png,Normal/masks/Normal-9238.png,True,False,299,299,L,111.34,4ebd5a379d5447bdd35045b4afccf6f6
Normal-9239.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9239.png,Normal/images/Normal-9239.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9239.png,Normal/masks/Normal-9239.png,True,False,299,299,L,118.59,20b31f6944c591a3131ed54b52836091
Normal-924.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-924.png,Normal/images/Normal-924.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-924.png,Normal/masks/Normal-924.png,True,False,299,299,L,100.54,1d2b5f86637c8102b1def8f315bd6967
Normal-9240.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9240.png,Normal/images/Normal-9240.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9240.png,Normal/masks/Normal-9240.png,True,False,299,299,L,161.89,a826439c889ab4c8767e2583f1d102cb
Normal-9241.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9241.png,Normal/images/Normal-9241.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9241.png,Normal/masks/Normal-9241.png,True,False,299,299,L,108.98,8db8b1831fd3919acad8887e6f11e6a6
Normal-9242.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9242.png,Normal/images/Normal-9242.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9242.png,Normal/masks/Normal-9242.png,True,False,299,299,L,151.39,857c7e51667a300bbc700caa4eedd9e7
Normal-9243.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9243.png,Normal/images/Normal-9243.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9243.png,Normal/masks/Normal-9243.png,True,False,299,299,L,150.01,4166114bb189892c38b3b5d8e558453a
Normal-9244.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9244.png,Normal/images/Normal-9244.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9244.png,Normal/masks/Normal-9244.png,True,False,299,299,L,118.73,5df98aea6dfba747b2d5d0a6eca91cee
Normal-9245.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9245.png,Normal/images/Normal-9245.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9245.png,Normal/masks/Normal-9245.png,True,False,299,299,L,122.03,b4a240015b0820dc3973922831ef5bc5
Normal-9246.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9246.png,Normal/images/Normal-9246.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9246.png,Normal/masks/Normal-9246.png,True,False,299,299,L,128.22,f293d68b5820760b93901175912b2f51
Normal-9247.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9247.png,Normal/images/Normal-9247.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9247.png,Normal/masks/Normal-9247.png,True,False,299,299,L,123.07,e18dcf258b6321565ee2dc9c22023e6c
Normal-9248.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9248.png,Normal/images/Normal-9248.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9248.png,Normal/masks/Normal-9248.png,True,False,299,299,L,104.25,33ffe29acd9240fa8bd49b82f9820147
Normal-9249.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9249.png,Normal/images/Normal-9249.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9249.png,Normal/masks/Normal-9249.png,True,False,299,299,L,136.49,d5235f10c828162101ccc42e3a86a537
Normal-925.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-925.png,Normal/images/Normal-925.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-925.png,Normal/masks/Normal-925.png,True,False,299,299,L,115.62,0670bc3f5bb7f4f28232b47f401b27c3
Normal-9250.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9250.png,Normal/images/Normal-9250.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9250.png,Normal/masks/Normal-9250.png,True,False,299,299,L,132.07,53ce204d9bc46249e5a8e397cde3ea56
Normal-9251.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9251.png,Normal/images/Normal-9251.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9251.png,Normal/masks/Normal-9251.png,True,False,299,299,L,109.42,8b811c4e5966ba7cda16c81bc486fbcc
Normal-9252.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9252.png,Normal/images/Normal-9252.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9252.png,Normal/masks/Normal-9252.png,True,False,299,299,L,100.75,4067dcb4d562ee4485e88a659c3cc8ae
Normal-9253.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9253.png,Normal/images/Normal-9253.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9253.png,Normal/masks/Normal-9253.png,True,False,299,299,L,142.51,60b46d8d7019cc66b5621c82797bdcf7
Normal-9254.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9254.png,Normal/images/Normal-9254.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9254.png,Normal/masks/Normal-9254.png,True,False,299,299,L,146.9,0af58c72fa71b625be77bf0ed2a03a5c
Normal-9255.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9255.png,Normal/images/Normal-9255.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9255.png,Normal/masks/Normal-9255.png,True,False,299,299,L,117.41,19ce944188d77bd07b76896dd1dd81ef
Normal-9256.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9256.png,Normal/images/Normal-9256.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9256.png,Normal/masks/Normal-9256.png,True,False,299,299,L,118.99,ace363cbd553bb65872b5802f6e4e43b
Normal-9257.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9257.png,Normal/images/Normal-9257.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9257.png,Normal/masks/Normal-9257.png,True,False,299,299,L,149.02,cc33b15025973a723cc387e19a80be06
Normal-9258.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9258.png,Normal/images/Normal-9258.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9258.png,Normal/masks/Normal-9258.png,True,False,299,299,L,138.58,5c75b3f7f2584dcba22a7c4e19ecb913
Normal-9259.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9259.png,Normal/images/Normal-9259.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9259.png,Normal/masks/Normal-9259.png,True,False,299,299,L,128.73,2fc3e82f548a73dcb3762d599c9aabd4
Normal-926.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-926.png,Normal/images/Normal-926.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-926.png,Normal/masks/Normal-926.png,True,False,299,299,L,119.92,b45f6e4403975781014f7ee2e1b5acdf
Normal-9260.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9260.png,Normal/images/Normal-9260.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9260.png,Normal/masks/Normal-9260.png,True,False,299,299,L,164.07,bc49fa5806eda0bbb1991954ddfd9876
Normal-9261.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9261.png,Normal/images/Normal-9261.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9261.png,Normal/masks/Normal-9261.png,True,False,299,299,L,150.03,e791b704c34d8b36ba263f66b5df573d
Normal-9262.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9262.png,Normal/images/Normal-9262.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9262.png,Normal/masks/Normal-9262.png,True,False,299,299,L,96.37,361453298b0eaf6232f2ec24766d9aaf
Normal-9263.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9263.png,Normal/images/Normal-9263.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9263.png,Normal/masks/Normal-9263.png,True,False,299,299,L,175.05,41d01b3017ee92760956b402b75a76af
Normal-9264.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9264.png,Normal/images/Normal-9264.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9264.png,Normal/masks/Normal-9264.png,True,False,299,299,L,129.82,9c8411274b5849f6580706a88cd94938
Normal-9265.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9265.png,Normal/images/Normal-9265.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9265.png,Normal/masks/Normal-9265.png,True,False,299,299,L,124.12,cafc09316356a093376142cd5b1ff4cd
Normal-9266.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9266.png,Normal/images/Normal-9266.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9266.png,Normal/masks/Normal-9266.png,True,False,299,299,L,160.91,29f8c521a6b807df0147a30d76db527a
Normal-9267.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9267.png,Normal/images/Normal-9267.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9267.png,Normal/masks/Normal-9267.png,True,False,299,299,L,125.17,93701cd042143586712e4b92e978ece1
Normal-9268.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9268.png,Normal/images/Normal-9268.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9268.png,Normal/masks/Normal-9268.png,True,False,299,299,L,104.59,df24cf2afebbbe50c491e32b3d202332
Normal-9269.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9269.png,Normal/images/Normal-9269.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9269.png,Normal/masks/Normal-9269.png,True,False,299,299,L,118.37,7deb5139abb8cf62282ea822004964c3
Normal-927.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-927.png,Normal/images/Normal-927.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-927.png,Normal/masks/Normal-927.png,True,False,299,299,L,97.2,eaa4b690a7798a128cfe2ae28e4ead24
Normal-9270.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9270.png,Normal/images/Normal-9270.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9270.png,Normal/masks/Normal-9270.png,True,False,299,299,L,109.55,77a0346960bdb3a9e76cec9bc28ca90e
Normal-9271.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9271.png,Normal/images/Normal-9271.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9271.png,Normal/masks/Normal-9271.png,True,False,299,299,L,145.08,2222ad9e1a564eaccd875d5e99ab9188
Normal-9272.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9272.png,Normal/images/Normal-9272.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9272.png,Normal/masks/Normal-9272.png,True,False,299,299,L,117.14,d9b469599789a08c76f8609e85c9ef2c
Normal-9273.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9273.png,Normal/images/Normal-9273.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9273.png,Normal/masks/Normal-9273.png,True,False,299,299,L,167.13,100d7a29471d4008cad8525fce01ea5f
Normal-9274.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9274.png,Normal/images/Normal-9274.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9274.png,Normal/masks/Normal-9274.png,True,False,299,299,L,160.59,e79c5423c6a60fc67e6f66166dc5f6c9
Normal-9275.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9275.png,Normal/images/Normal-9275.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9275.png,Normal/masks/Normal-9275.png,True,False,299,299,L,137.1,d8d1d7080b8437b367e0743d80ff2991
Normal-9276.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9276.png,Normal/images/Normal-9276.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9276.png,Normal/masks/Normal-9276.png,True,False,299,299,L,106.87,afdf85dcc7b7d62ddf2a6f3189c7be85
Normal-9277.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9277.png,Normal/images/Normal-9277.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9277.png,Normal/masks/Normal-9277.png,True,False,299,299,L,160.19,a1a80c834811f86b64539bc537af237d
Normal-9278.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9278.png,Normal/images/Normal-9278.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9278.png,Normal/masks/Normal-9278.png,True,False,299,299,L,100.81,c47b6b2215399ee1d3074ee8fca370af
Normal-9279.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9279.png,Normal/images/Normal-9279.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9279.png,Normal/masks/Normal-9279.png,True,False,299,299,L,89.68,e15317a489b56b519c496ad7e4ebe929
Normal-928.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-928.png,Normal/images/Normal-928.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-928.png,Normal/masks/Normal-928.png,True,False,299,299,L,122.74,a9da5844df2788f75e259e694e884291
Normal-9280.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9280.png,Normal/images/Normal-9280.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9280.png,Normal/masks/Normal-9280.png,True,False,299,299,L,145.77,c3dfe6829ddbda1647245daeaf984e9b
Normal-9281.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9281.png,Normal/images/Normal-9281.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9281.png,Normal/masks/Normal-9281.png,True,False,299,299,L,121.85,83afdeb6f0ccd1b8ea0648ab1fe583b7
Normal-9282.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9282.png,Normal/images/Normal-9282.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9282.png,Normal/masks/Normal-9282.png,True,False,299,299,L,95.94,6b5dcb172ca65977f3b5968e45811afd
Normal-9283.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9283.png,Normal/images/Normal-9283.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9283.png,Normal/masks/Normal-9283.png,True,False,299,299,L,144.25,f44db8cbe88796a034b4fa07f6507e32
Normal-9284.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9284.png,Normal/images/Normal-9284.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9284.png,Normal/masks/Normal-9284.png,True,False,299,299,L,180.78,fa29a0f1c01ec163d4f801ecf92b7d9b
Normal-9285.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9285.png,Normal/images/Normal-9285.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9285.png,Normal/masks/Normal-9285.png,True,False,299,299,L,137.68,c32c1e4a0033d60a66848edb807a0fed
Normal-9286.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9286.png,Normal/images/Normal-9286.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9286.png,Normal/masks/Normal-9286.png,True,False,299,299,L,86.02,98a199cc5c1e8ae5c4c77910c700c82f
Normal-9287.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9287.png,Normal/images/Normal-9287.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9287.png,Normal/masks/Normal-9287.png,True,False,299,299,L,100.61,600add205b85b9660e6c62e0c05d102a
Normal-9288.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9288.png,Normal/images/Normal-9288.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9288.png,Normal/masks/Normal-9288.png,True,False,299,299,L,174.56,88f032df395ca6baaa0f75042535125f
Normal-9289.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9289.png,Normal/images/Normal-9289.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9289.png,Normal/masks/Normal-9289.png,True,False,299,299,L,115.83,f24880dabfe4fb8a452679675d97c1db
Normal-929.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-929.png,Normal/images/Normal-929.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-929.png,Normal/masks/Normal-929.png,True,False,299,299,L,110.64,e872601bacc99839afc8e005d0411154
Normal-9290.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9290.png,Normal/images/Normal-9290.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9290.png,Normal/masks/Normal-9290.png,True,False,299,299,L,115.66,28eb1ea17343e7c087437b3054536a28
Normal-9291.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9291.png,Normal/images/Normal-9291.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9291.png,Normal/masks/Normal-9291.png,True,False,299,299,L,131.16,40b7afaddc5d7780fcc038d8157bdff8
Normal-9292.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9292.png,Normal/images/Normal-9292.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9292.png,Normal/masks/Normal-9292.png,True,False,299,299,L,118.82,a81ab8bca10fc982abaa54a1c5d795c4
Normal-9293.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9293.png,Normal/images/Normal-9293.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9293.png,Normal/masks/Normal-9293.png,True,False,299,299,L,104.14,ed9ce59081b782e33c17563ee562b912
Normal-9294.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9294.png,Normal/images/Normal-9294.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9294.png,Normal/masks/Normal-9294.png,True,False,299,299,L,140.66,9b633a02e477ee8617d976808644bb51
Normal-9295.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9295.png,Normal/images/Normal-9295.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9295.png,Normal/masks/Normal-9295.png,True,False,299,299,L,182.49,6a6f2d320cf229888fe47f8742c2a53a
Normal-9296.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9296.png,Normal/images/Normal-9296.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9296.png,Normal/masks/Normal-9296.png,True,False,299,299,L,108.17,b54a05a97f8b9b35a716efd1c8840a60
Normal-9297.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9297.png,Normal/images/Normal-9297.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9297.png,Normal/masks/Normal-9297.png,True,False,299,299,L,113.29,5c98619d2d4ec614a28774eb56c297fd
Normal-9298.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9298.png,Normal/images/Normal-9298.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9298.png,Normal/masks/Normal-9298.png,True,False,299,299,L,118.36,46c9f63f7edaa4b76b564e732ee86893
Normal-9299.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9299.png,Normal/images/Normal-9299.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9299.png,Normal/masks/Normal-9299.png,True,False,299,299,L,107.43,1ca472d7baf725965722778a7494e9bd
Normal-93.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-93.png,Normal/images/Normal-93.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-93.png,Normal/masks/Normal-93.png,True,False,299,299,L,154.89,024519648c067c5ec38535413c4787c5
Normal-930.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-930.png,Normal/images/Normal-930.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-930.png,Normal/masks/Normal-930.png,True,False,299,299,L,108.52,7bcea9cf73a10d2c794f017982711ab5
Normal-9300.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9300.png,Normal/images/Normal-9300.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9300.png,Normal/masks/Normal-9300.png,True,False,299,299,L,165.7,96239cbf7d105ff37d214df128e541da
Normal-9301.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9301.png,Normal/images/Normal-9301.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9301.png,Normal/masks/Normal-9301.png,True,False,299,299,L,174.15,4af2ddc3a858bc888a61411f115b9852
Normal-9302.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9302.png,Normal/images/Normal-9302.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9302.png,Normal/masks/Normal-9302.png,True,False,299,299,L,128.43,656232bbdedb598bfbc9a28ea867e092
Normal-9303.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9303.png,Normal/images/Normal-9303.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9303.png,Normal/masks/Normal-9303.png,True,False,299,299,L,121.91,7146498c12a77505a8d76390155ea38e
Normal-9304.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9304.png,Normal/images/Normal-9304.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9304.png,Normal/masks/Normal-9304.png,True,False,299,299,L,121.87,aa06766ffd0dca79e600f39646e7033f
Normal-9305.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9305.png,Normal/images/Normal-9305.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9305.png,Normal/masks/Normal-9305.png,True,False,299,299,L,123.37,1ef28cc64b218c5dfea9dcd706c391bf
Normal-9306.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9306.png,Normal/images/Normal-9306.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9306.png,Normal/masks/Normal-9306.png,True,False,299,299,L,126.95,caac964b90ed5496b69d5997d238e178
Normal-9307.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9307.png,Normal/images/Normal-9307.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9307.png,Normal/masks/Normal-9307.png,True,False,299,299,L,115.46,ccc3cf5d83b3e2a5b666cf7763ae4cf7
Normal-9308.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9308.png,Normal/images/Normal-9308.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9308.png,Normal/masks/Normal-9308.png,True,False,299,299,L,126.71,d3ebfcfe9d9c242fb15b710691e4fdf1
Normal-9309.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9309.png,Normal/images/Normal-9309.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9309.png,Normal/masks/Normal-9309.png,True,False,299,299,L,174.58,e235247e753d1811f3afe56843086e24
Normal-931.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-931.png,Normal/images/Normal-931.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-931.png,Normal/masks/Normal-931.png,True,False,299,299,L,118.89,aa1f8511a99313135d405b1c3d4a3c77
Normal-9310.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9310.png,Normal/images/Normal-9310.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9310.png,Normal/masks/Normal-9310.png,True,False,299,299,L,104.31,6b3863a3fbd19eab93b420564368a344
Normal-9311.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9311.png,Normal/images/Normal-9311.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9311.png,Normal/masks/Normal-9311.png,True,False,299,299,L,116.36,4e1708c90835b27aa3c775d53268df93
Normal-9312.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9312.png,Normal/images/Normal-9312.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9312.png,Normal/masks/Normal-9312.png,True,False,299,299,L,124.69,892ecfd4c684d7158b73565ad6bdc045
Normal-9313.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9313.png,Normal/images/Normal-9313.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9313.png,Normal/masks/Normal-9313.png,True,False,299,299,L,112.95,ee41480ba7fb9dfd00f4327a52b900ab
Normal-9314.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9314.png,Normal/images/Normal-9314.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9314.png,Normal/masks/Normal-9314.png,True,False,299,299,L,131.49,1efc248b576cb107a4d90baa87919eed
Normal-9315.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9315.png,Normal/images/Normal-9315.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9315.png,Normal/masks/Normal-9315.png,True,False,299,299,L,104.89,0290642da21c4362aed8dea96c0cde35
Normal-9316.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9316.png,Normal/images/Normal-9316.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9316.png,Normal/masks/Normal-9316.png,True,False,299,299,L,141.51,b0265653196efb37a5e0abfbfcc759bc
Normal-9317.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9317.png,Normal/images/Normal-9317.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9317.png,Normal/masks/Normal-9317.png,True,False,299,299,L,119.8,38b7d0dedfde9dca9b7b40904ee273b7
Normal-9318.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9318.png,Normal/images/Normal-9318.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9318.png,Normal/masks/Normal-9318.png,True,False,299,299,L,112.14,ad2b2f5c5c57c510d25deba56c1dd077
Normal-9319.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9319.png,Normal/images/Normal-9319.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9319.png,Normal/masks/Normal-9319.png,True,False,299,299,L,121.01,5a95448fbd664643fd8cc090197996e5
Normal-932.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-932.png,Normal/images/Normal-932.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-932.png,Normal/masks/Normal-932.png,True,False,299,299,L,117.48,3c21b3641f5c1c8c8948d400f882f9c6
Normal-9320.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9320.png,Normal/images/Normal-9320.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9320.png,Normal/masks/Normal-9320.png,True,False,299,299,L,126.99,d9545242c191d99bb74bec54f4e50313
Normal-9321.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9321.png,Normal/images/Normal-9321.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9321.png,Normal/masks/Normal-9321.png,True,False,299,299,L,120.48,92d5c86faa2e9b97a05db7db65fd750e
Normal-9322.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9322.png,Normal/images/Normal-9322.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9322.png,Normal/masks/Normal-9322.png,True,False,299,299,L,103.32,8a178410b28034ecd6f361cb7b931fef
Normal-9323.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9323.png,Normal/images/Normal-9323.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9323.png,Normal/masks/Normal-9323.png,True,False,299,299,L,144.26,29a19857ab2217a9149844125531f766
Normal-9324.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9324.png,Normal/images/Normal-9324.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9324.png,Normal/masks/Normal-9324.png,True,False,299,299,L,103.28,19947e74dea9c2370e53bca08105a98b
Normal-9325.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9325.png,Normal/images/Normal-9325.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9325.png,Normal/masks/Normal-9325.png,True,False,299,299,L,134.58,9fe66c32276690eeef7d6025c316810f
Normal-9326.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9326.png,Normal/images/Normal-9326.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9326.png,Normal/masks/Normal-9326.png,True,False,299,299,L,110.25,e74aeedfa7c6fff26dd07dff5f52a79d
Normal-9327.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9327.png,Normal/images/Normal-9327.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9327.png,Normal/masks/Normal-9327.png,True,False,299,299,L,145.54,89631be7a9ec092200053b99670519a4
Normal-9328.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9328.png,Normal/images/Normal-9328.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9328.png,Normal/masks/Normal-9328.png,True,False,299,299,L,175.98,3a2d7004fc1a2a5f0069077def35d872
Normal-9329.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9329.png,Normal/images/Normal-9329.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9329.png,Normal/masks/Normal-9329.png,True,False,299,299,L,136.09,b4025c79c49917612cc2dcd6b5b9856d
Normal-933.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-933.png,Normal/images/Normal-933.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-933.png,Normal/masks/Normal-933.png,True,False,299,299,L,91.41,d4b9b0e5f71ad0e387d966e167c0c417
Normal-9330.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9330.png,Normal/images/Normal-9330.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9330.png,Normal/masks/Normal-9330.png,True,False,299,299,L,117.01,6ac02d56d20205840690e0ca60bc7422
Normal-9331.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9331.png,Normal/images/Normal-9331.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9331.png,Normal/masks/Normal-9331.png,True,False,299,299,L,105.51,c9165a126b61d55aaefef1c74235632d
Normal-9332.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9332.png,Normal/images/Normal-9332.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9332.png,Normal/masks/Normal-9332.png,True,False,299,299,L,122.99,50d4048b11315a056094c2c6a5c59142
Normal-9333.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9333.png,Normal/images/Normal-9333.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9333.png,Normal/masks/Normal-9333.png,True,False,299,299,L,151.81,72c032030b10f82340eb56a780aaf1ef
Normal-9334.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9334.png,Normal/images/Normal-9334.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9334.png,Normal/masks/Normal-9334.png,True,False,299,299,L,103.86,d2836420228f4e8768e4b284107d4779
Normal-9335.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9335.png,Normal/images/Normal-9335.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9335.png,Normal/masks/Normal-9335.png,True,False,299,299,L,122.35,1e10f557e10b124911bd8f5d32bd4a29
Normal-9336.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9336.png,Normal/images/Normal-9336.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9336.png,Normal/masks/Normal-9336.png,True,False,299,299,L,120.62,21e93f4b6a1e00c9727558a8a37d3736
Normal-9337.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9337.png,Normal/images/Normal-9337.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9337.png,Normal/masks/Normal-9337.png,True,False,299,299,L,143.02,f4c475a96bd6466d67793f91e1d48966
Normal-9338.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9338.png,Normal/images/Normal-9338.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9338.png,Normal/masks/Normal-9338.png,True,False,299,299,L,186.44,b1c16c0d127b402669f43e5d15cb1571
Normal-9339.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9339.png,Normal/images/Normal-9339.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9339.png,Normal/masks/Normal-9339.png,True,False,299,299,L,105.89,3860d4328cb61a0593de7aec16c64c52
Normal-934.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-934.png,Normal/images/Normal-934.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-934.png,Normal/masks/Normal-934.png,True,False,299,299,L,133.84,84bab56eddfbf84472f6163ec85a8c79
Normal-9340.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9340.png,Normal/images/Normal-9340.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9340.png,Normal/masks/Normal-9340.png,True,False,299,299,L,133.83,780113f3884d10fc899b59608596953b
Normal-9341.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9341.png,Normal/images/Normal-9341.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9341.png,Normal/masks/Normal-9341.png,True,False,299,299,L,147.13,0feea335ea396e6599edaaed95fb8f9f
Normal-9342.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9342.png,Normal/images/Normal-9342.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9342.png,Normal/masks/Normal-9342.png,True,False,299,299,L,137.21,8def5d147c6993b6c5f2ca92c7de7ef5
Normal-9343.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9343.png,Normal/images/Normal-9343.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9343.png,Normal/masks/Normal-9343.png,True,False,299,299,L,133.3,7001539d9054de27faa17b4a6ec28c60
Normal-9344.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9344.png,Normal/images/Normal-9344.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9344.png,Normal/masks/Normal-9344.png,True,False,299,299,L,154.16,c220dc59b13573a3765028f17ebfb04b
Normal-9345.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9345.png,Normal/images/Normal-9345.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9345.png,Normal/masks/Normal-9345.png,True,False,299,299,L,133.82,001d1562ea33227be0f24ec4eebb99a3
Normal-9346.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9346.png,Normal/images/Normal-9346.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9346.png,Normal/masks/Normal-9346.png,True,False,299,299,L,120.6,fc47b7ebf8c77e7037e660f01e805be7
Normal-9347.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9347.png,Normal/images/Normal-9347.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9347.png,Normal/masks/Normal-9347.png,True,False,299,299,L,167.53,de5fa17dde840b229a295a61cdd91c67
Normal-9348.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9348.png,Normal/images/Normal-9348.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9348.png,Normal/masks/Normal-9348.png,True,False,299,299,L,125.08,4b82c6880d7a84d18ab4cc159ec801a7
Normal-9349.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9349.png,Normal/images/Normal-9349.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9349.png,Normal/masks/Normal-9349.png,True,False,299,299,L,125.13,f72c98df79c96ec4096cefb70cd9d4a9
Normal-935.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-935.png,Normal/images/Normal-935.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-935.png,Normal/masks/Normal-935.png,True,False,299,299,L,121.51,363b00660060fe9137863a4f1ef4118b
Normal-9350.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9350.png,Normal/images/Normal-9350.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9350.png,Normal/masks/Normal-9350.png,True,False,299,299,L,95.27,994e7a7910e01b48e0a55a9eb3fc4082
Normal-9351.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9351.png,Normal/images/Normal-9351.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9351.png,Normal/masks/Normal-9351.png,True,False,299,299,L,109.71,404db5c3085302e398e48f29dd3f20b3
Normal-9352.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9352.png,Normal/images/Normal-9352.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9352.png,Normal/masks/Normal-9352.png,True,False,299,299,L,157.5,f97597ef41a6e5a88a490bc0a959c153
Normal-9353.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9353.png,Normal/images/Normal-9353.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9353.png,Normal/masks/Normal-9353.png,True,False,299,299,L,136.71,66b8f9bf1c31a63b1b948d07f15d2649
Normal-9354.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9354.png,Normal/images/Normal-9354.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9354.png,Normal/masks/Normal-9354.png,True,False,299,299,L,98.22,af2abc1baa5d6c5bb319f7ca05cac87b
Normal-9355.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9355.png,Normal/images/Normal-9355.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9355.png,Normal/masks/Normal-9355.png,True,False,299,299,L,110.46,7ae1c6d058b0070851507036da25fb15
Normal-9356.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9356.png,Normal/images/Normal-9356.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9356.png,Normal/masks/Normal-9356.png,True,False,299,299,L,118.74,5e1c38283c433d7d2ba9057939e0ebe7
Normal-9357.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9357.png,Normal/images/Normal-9357.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9357.png,Normal/masks/Normal-9357.png,True,False,299,299,L,125.49,bad5050bbd49a5da6c2d9d32d038d13c
Normal-9358.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9358.png,Normal/images/Normal-9358.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9358.png,Normal/masks/Normal-9358.png,True,False,299,299,L,130.18,8ef316ef877f5eaa4b27cb84a437ecb5
Normal-9359.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9359.png,Normal/images/Normal-9359.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9359.png,Normal/masks/Normal-9359.png,True,False,299,299,L,111.98,f261463899a136933943a10849ca47d7
Normal-936.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-936.png,Normal/images/Normal-936.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-936.png,Normal/masks/Normal-936.png,True,False,299,299,L,144.72,bb9d7e40c1d8d6c358c31032c749fd55
Normal-9360.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9360.png,Normal/images/Normal-9360.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9360.png,Normal/masks/Normal-9360.png,True,False,299,299,L,167.84,fe24cb75914865a551ffb41231b2423b
Normal-9361.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9361.png,Normal/images/Normal-9361.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9361.png,Normal/masks/Normal-9361.png,True,False,299,299,L,154.91,50e3ca2914c6582346d7d10fca00ca48
Normal-9362.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9362.png,Normal/images/Normal-9362.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9362.png,Normal/masks/Normal-9362.png,True,False,299,299,L,114.68,bf1e6485b52cfc21a3ee95f43cd2b5dc
Normal-9363.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9363.png,Normal/images/Normal-9363.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9363.png,Normal/masks/Normal-9363.png,True,False,299,299,L,123.24,6973a7129b8825f8cc0d68405355e7c6
Normal-9364.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9364.png,Normal/images/Normal-9364.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9364.png,Normal/masks/Normal-9364.png,True,False,299,299,L,152.08,fbac0dd51b968253a6e0d26a5406bdd6
Normal-9365.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9365.png,Normal/images/Normal-9365.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9365.png,Normal/masks/Normal-9365.png,True,False,299,299,L,113.77,fdc54cea3c1de6ded3be00df3c7734df
Normal-9366.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9366.png,Normal/images/Normal-9366.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9366.png,Normal/masks/Normal-9366.png,True,False,299,299,L,116.61,5936077e422714279591b64f83b5ef03
Normal-9367.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9367.png,Normal/images/Normal-9367.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9367.png,Normal/masks/Normal-9367.png,True,False,299,299,L,166.5,5f4bff1941310b887691fb9146001e1c
Normal-9368.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9368.png,Normal/images/Normal-9368.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9368.png,Normal/masks/Normal-9368.png,True,False,299,299,L,116.69,fc7075760692f2ca88bbad71f36bbc88
Normal-9369.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9369.png,Normal/images/Normal-9369.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9369.png,Normal/masks/Normal-9369.png,True,False,299,299,L,153.4,c16b4531ff026d69b959918c686de5e7
Normal-937.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-937.png,Normal/images/Normal-937.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-937.png,Normal/masks/Normal-937.png,True,False,299,299,L,137.47,57bcd7dee76fa811bdecf1a5778b8992
Normal-9370.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9370.png,Normal/images/Normal-9370.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9370.png,Normal/masks/Normal-9370.png,True,False,299,299,L,111.66,c040634d74fc57f150c230da2b029819
Normal-9371.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9371.png,Normal/images/Normal-9371.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9371.png,Normal/masks/Normal-9371.png,True,False,299,299,L,170.99,c7447ae22ca9723c8d7141f282a41fab
Normal-9372.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9372.png,Normal/images/Normal-9372.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9372.png,Normal/masks/Normal-9372.png,True,False,299,299,L,157.63,68170c7556e45a765db80340cfece463
Normal-9373.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9373.png,Normal/images/Normal-9373.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9373.png,Normal/masks/Normal-9373.png,True,False,299,299,L,122.5,cc9d912c6607d86c867091af19dc2c1d
Normal-9374.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9374.png,Normal/images/Normal-9374.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9374.png,Normal/masks/Normal-9374.png,True,False,299,299,L,145.0,3de372121a6ed1c9891b4b85bc39f887
Normal-9375.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9375.png,Normal/images/Normal-9375.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9375.png,Normal/masks/Normal-9375.png,True,False,299,299,L,162.5,e6d93305b84da78e5512838cf9f7e33f
Normal-9376.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9376.png,Normal/images/Normal-9376.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9376.png,Normal/masks/Normal-9376.png,True,False,299,299,L,113.46,f4cabae4f77903cb0d05072c502108e4
Normal-9377.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9377.png,Normal/images/Normal-9377.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9377.png,Normal/masks/Normal-9377.png,True,False,299,299,L,112.56,f89db178b8b13b9c9a05ea67edabbdc9
Normal-9378.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9378.png,Normal/images/Normal-9378.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9378.png,Normal/masks/Normal-9378.png,True,False,299,299,L,132.64,841b8cc61310c169b160bf6ef696d312
Normal-9379.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9379.png,Normal/images/Normal-9379.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9379.png,Normal/masks/Normal-9379.png,True,False,299,299,L,149.55,dfcbf897321ead746285fd9e93d56a5c
Normal-938.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-938.png,Normal/images/Normal-938.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-938.png,Normal/masks/Normal-938.png,True,False,299,299,L,130.89,c8ce8af14e214801664c555f2e9a80b1
Normal-9380.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9380.png,Normal/images/Normal-9380.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9380.png,Normal/masks/Normal-9380.png,True,False,299,299,L,155.99,b76cfb5e848209efd4f8f696db3e4a57
Normal-9381.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9381.png,Normal/images/Normal-9381.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9381.png,Normal/masks/Normal-9381.png,True,False,299,299,L,137.68,d37ab9ef668fc1a444315cbf3cee6533
Normal-9382.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9382.png,Normal/images/Normal-9382.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9382.png,Normal/masks/Normal-9382.png,True,False,299,299,L,113.02,d0bb806692e79157725737b6282ca419
Normal-9383.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9383.png,Normal/images/Normal-9383.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9383.png,Normal/masks/Normal-9383.png,True,False,299,299,L,140.65,f066c117532dc50fabf205cd8a87538f
Normal-9384.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9384.png,Normal/images/Normal-9384.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9384.png,Normal/masks/Normal-9384.png,True,False,299,299,L,102.99,71965f6b0533c6de5835493cdb3a4082
Normal-9385.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9385.png,Normal/images/Normal-9385.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9385.png,Normal/masks/Normal-9385.png,True,False,299,299,L,111.32,ca9bd5afb9d6b9cdb1b05e1a904eddd4
Normal-9386.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9386.png,Normal/images/Normal-9386.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9386.png,Normal/masks/Normal-9386.png,True,False,299,299,L,94.23,1bfc373bf311259518b7e02d9312b376
Normal-9387.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9387.png,Normal/images/Normal-9387.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9387.png,Normal/masks/Normal-9387.png,True,False,299,299,L,119.28,3f9dd3b28ac8ec65e6c350d69c5fbc0a
Normal-9388.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9388.png,Normal/images/Normal-9388.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9388.png,Normal/masks/Normal-9388.png,True,False,299,299,L,166.44,afeef99ee07748be14b50399a0af6f13
Normal-9389.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9389.png,Normal/images/Normal-9389.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9389.png,Normal/masks/Normal-9389.png,True,False,299,299,L,126.51,ce59f41ffc633c03f9eb1b74be0b7ddd
Normal-939.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-939.png,Normal/images/Normal-939.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-939.png,Normal/masks/Normal-939.png,True,False,299,299,L,122.61,5d4250d3335a8734fe5ac5247c72d6d7
Normal-9390.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9390.png,Normal/images/Normal-9390.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9390.png,Normal/masks/Normal-9390.png,True,False,299,299,L,145.26,912eff593a0cefd836fb5f7dd1421677
Normal-9391.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9391.png,Normal/images/Normal-9391.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9391.png,Normal/masks/Normal-9391.png,True,False,299,299,L,129.19,5374e435c9f5b49b1b07b18f70526e44
Normal-9392.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9392.png,Normal/images/Normal-9392.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9392.png,Normal/masks/Normal-9392.png,True,False,299,299,L,113.06,3d0ec53e4e64e92fc1e820858ac11174
Normal-9393.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9393.png,Normal/images/Normal-9393.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9393.png,Normal/masks/Normal-9393.png,True,False,299,299,L,152.86,5f533e1fbdc0e1c5078e50b6a4ae0717
Normal-9394.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9394.png,Normal/images/Normal-9394.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9394.png,Normal/masks/Normal-9394.png,True,False,299,299,L,126.9,22e48f7c8a8a257e925aeeb6d09d39c2
Normal-9395.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9395.png,Normal/images/Normal-9395.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9395.png,Normal/masks/Normal-9395.png,True,False,299,299,L,103.07,53188b308643858ec1bd0e935b7ecac2
Normal-9396.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9396.png,Normal/images/Normal-9396.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9396.png,Normal/masks/Normal-9396.png,True,False,299,299,L,122.43,6c8415412ea3a87d2edbfb8c45379969
Normal-9397.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9397.png,Normal/images/Normal-9397.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9397.png,Normal/masks/Normal-9397.png,True,False,299,299,L,130.05,d9d5941c39db2d6d8b5f171860deee12
Normal-9398.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9398.png,Normal/images/Normal-9398.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9398.png,Normal/masks/Normal-9398.png,True,False,299,299,L,98.46,5d4f3e27f7ef53a79c16325700b3c0a8
Normal-9399.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9399.png,Normal/images/Normal-9399.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9399.png,Normal/masks/Normal-9399.png,True,False,299,299,L,148.37,48d4cfed22292220aeaad72899c83da3
Normal-94.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-94.png,Normal/images/Normal-94.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-94.png,Normal/masks/Normal-94.png,True,False,299,299,L,90.13,723f07827e9ced699039cb4df5988d5d
Normal-940.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-940.png,Normal/images/Normal-940.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-940.png,Normal/masks/Normal-940.png,True,False,299,299,L,109.25,09c56a4fea7e54f36328ad293e26a975
Normal-9400.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9400.png,Normal/images/Normal-9400.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9400.png,Normal/masks/Normal-9400.png,True,False,299,299,L,104.76,b912c30172c595dff609a599ff9949f6
Normal-9401.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9401.png,Normal/images/Normal-9401.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9401.png,Normal/masks/Normal-9401.png,True,False,299,299,L,165.44,46cd7f2e2ff6c82c0df0570d53012005
Normal-9402.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9402.png,Normal/images/Normal-9402.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9402.png,Normal/masks/Normal-9402.png,True,False,299,299,L,122.46,7706a542d3fbcef009bcb7752bb247a9
Normal-9403.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9403.png,Normal/images/Normal-9403.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9403.png,Normal/masks/Normal-9403.png,True,False,299,299,L,122.53,ea3cefb3a14a795d74622503937b931a
Normal-9404.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9404.png,Normal/images/Normal-9404.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9404.png,Normal/masks/Normal-9404.png,True,False,299,299,L,124.68,062650b49d0d5073b78965e260319750
Normal-9405.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9405.png,Normal/images/Normal-9405.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9405.png,Normal/masks/Normal-9405.png,True,False,299,299,L,114.71,a75c90064652b4b62d8ab9a6bd06b9b3
Normal-9406.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9406.png,Normal/images/Normal-9406.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9406.png,Normal/masks/Normal-9406.png,True,False,299,299,L,123.91,75a9ebb6b4a3b02545683b51147d95c8
Normal-9407.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9407.png,Normal/images/Normal-9407.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9407.png,Normal/masks/Normal-9407.png,True,False,299,299,L,169.23,e6c96a7f1f0e1d9e6aaedcc080149a62
Normal-9408.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9408.png,Normal/images/Normal-9408.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9408.png,Normal/masks/Normal-9408.png,True,False,299,299,L,130.31,4f544be753a382b6538888a7f300c635
Normal-9409.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9409.png,Normal/images/Normal-9409.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9409.png,Normal/masks/Normal-9409.png,True,False,299,299,L,109.39,e3a8a4a5cc2c0ae705c24e0846cf33f3
Normal-941.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-941.png,Normal/images/Normal-941.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-941.png,Normal/masks/Normal-941.png,True,False,299,299,L,134.29,7e845f0f0a658f7bd7f3f36b03617e9b
Normal-9410.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9410.png,Normal/images/Normal-9410.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9410.png,Normal/masks/Normal-9410.png,True,False,299,299,L,152.93,33631d9cdc529c037dcca6fa22979aa4
Normal-9411.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9411.png,Normal/images/Normal-9411.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9411.png,Normal/masks/Normal-9411.png,True,False,299,299,L,123.29,3d1a85b54cf3ec34fee8f51872525d28
Normal-9412.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9412.png,Normal/images/Normal-9412.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9412.png,Normal/masks/Normal-9412.png,True,False,299,299,L,108.11,434f17e4f0079fb481eb40ebca9f38c7
Normal-9413.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9413.png,Normal/images/Normal-9413.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9413.png,Normal/masks/Normal-9413.png,True,False,299,299,L,133.6,1af9f57ec4581d85077153f78e656fdb
Normal-9414.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9414.png,Normal/images/Normal-9414.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9414.png,Normal/masks/Normal-9414.png,True,False,299,299,L,124.28,c23ce59b4fa988f8f4d370f839488d5a
Normal-9415.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9415.png,Normal/images/Normal-9415.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9415.png,Normal/masks/Normal-9415.png,True,False,299,299,L,119.08,d32cbc443d03e818b3a54f5fc3e8edd7
Normal-9416.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9416.png,Normal/images/Normal-9416.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9416.png,Normal/masks/Normal-9416.png,True,False,299,299,L,118.78,c6c616579704de46139e82bdfac75731
Normal-9417.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9417.png,Normal/images/Normal-9417.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9417.png,Normal/masks/Normal-9417.png,True,False,299,299,L,144.71,89701e418845cf9b2edb22135021f180
Normal-9418.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9418.png,Normal/images/Normal-9418.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9418.png,Normal/masks/Normal-9418.png,True,False,299,299,L,106.22,4657fd85b9643c2fd5e05b3e49bce860
Normal-9419.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9419.png,Normal/images/Normal-9419.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9419.png,Normal/masks/Normal-9419.png,True,False,299,299,L,109.48,956004b96c76c803e675fe20d41f9a8f
Normal-942.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-942.png,Normal/images/Normal-942.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-942.png,Normal/masks/Normal-942.png,True,False,299,299,L,126.95,b81866f2aee5178faef507b5e5b3168a
Normal-9420.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9420.png,Normal/images/Normal-9420.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9420.png,Normal/masks/Normal-9420.png,True,False,299,299,L,133.81,33707aae31b05f4eb6900f7a54976776
Normal-9421.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9421.png,Normal/images/Normal-9421.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9421.png,Normal/masks/Normal-9421.png,True,False,299,299,L,134.71,881a85391df759366174d3217fbb7e87
Normal-9422.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9422.png,Normal/images/Normal-9422.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9422.png,Normal/masks/Normal-9422.png,True,False,299,299,L,133.1,37e211758b0433bd2eb35351e5465591
Normal-9423.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9423.png,Normal/images/Normal-9423.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9423.png,Normal/masks/Normal-9423.png,True,False,299,299,L,166.52,b0f9e8279c942a118b0e41ebe868dde0
Normal-9424.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9424.png,Normal/images/Normal-9424.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9424.png,Normal/masks/Normal-9424.png,True,False,299,299,L,155.84,7eb088538da306532545496c9c37a7c5
Normal-9425.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9425.png,Normal/images/Normal-9425.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9425.png,Normal/masks/Normal-9425.png,True,False,299,299,L,124.51,5d293bdb6e7654b2c63ddf7c19d8feb5
Normal-9426.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9426.png,Normal/images/Normal-9426.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9426.png,Normal/masks/Normal-9426.png,True,False,299,299,L,136.45,27d76952bbc6afd2a615e796a96c43e1
Normal-9427.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9427.png,Normal/images/Normal-9427.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9427.png,Normal/masks/Normal-9427.png,True,False,299,299,L,123.61,b33266827137260c6085f616b30d160b
Normal-9428.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9428.png,Normal/images/Normal-9428.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9428.png,Normal/masks/Normal-9428.png,True,False,299,299,L,118.61,3cfe1ff9532c3ce93566081de875c369
Normal-9429.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9429.png,Normal/images/Normal-9429.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9429.png,Normal/masks/Normal-9429.png,True,False,299,299,L,112.36,380dcf18751252d0374a51204d81fee3
Normal-943.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-943.png,Normal/images/Normal-943.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-943.png,Normal/masks/Normal-943.png,True,False,299,299,L,133.99,1f573c8c9101fd7be1bda0e4abbe3083
Normal-9430.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9430.png,Normal/images/Normal-9430.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9430.png,Normal/masks/Normal-9430.png,True,False,299,299,L,139.75,fcd6fa012ab0e0394de6708a778d0846
Normal-9431.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9431.png,Normal/images/Normal-9431.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9431.png,Normal/masks/Normal-9431.png,True,False,299,299,L,136.92,e28875b6e1184f7a5322821d19e05e12
Normal-9432.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9432.png,Normal/images/Normal-9432.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9432.png,Normal/masks/Normal-9432.png,True,False,299,299,L,163.65,36908cd472add23b43b453d21a1de754
Normal-9433.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9433.png,Normal/images/Normal-9433.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9433.png,Normal/masks/Normal-9433.png,True,False,299,299,L,160.25,75b7ec8211ba134ee259370ed933f261
Normal-9434.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9434.png,Normal/images/Normal-9434.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9434.png,Normal/masks/Normal-9434.png,True,False,299,299,L,127.8,830e896369a476c7c720b6ebe8d03f1d
Normal-9435.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9435.png,Normal/images/Normal-9435.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9435.png,Normal/masks/Normal-9435.png,True,False,299,299,L,135.6,17d8b4828b9a18dd30b622dc945f41fc
Normal-9436.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9436.png,Normal/images/Normal-9436.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9436.png,Normal/masks/Normal-9436.png,True,False,299,299,L,156.18,159d614cf0c82e8fb418d5826cdcac95
Normal-9437.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9437.png,Normal/images/Normal-9437.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9437.png,Normal/masks/Normal-9437.png,True,False,299,299,L,128.65,7c2316b682b70b2f97fff2287bdc559a
Normal-9438.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9438.png,Normal/images/Normal-9438.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9438.png,Normal/masks/Normal-9438.png,True,False,299,299,L,144.72,cfe3869f55aa854c7ba7ec06b008d0cc
Normal-9439.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9439.png,Normal/images/Normal-9439.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9439.png,Normal/masks/Normal-9439.png,True,False,299,299,L,179.12,76dfa373862c5d5e441a4ffd12eb84d2
Normal-944.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-944.png,Normal/images/Normal-944.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-944.png,Normal/masks/Normal-944.png,True,False,299,299,L,116.94,de069cbf882ef2649d99d7d5aa4cedb0
Normal-9440.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9440.png,Normal/images/Normal-9440.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9440.png,Normal/masks/Normal-9440.png,True,False,299,299,L,112.72,a239e8ba459ae4d0414fb8b127d05be5
Normal-9441.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9441.png,Normal/images/Normal-9441.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9441.png,Normal/masks/Normal-9441.png,True,False,299,299,L,146.85,33ea33bc5bbc9a86760dfc8602d21b1a
Normal-9442.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9442.png,Normal/images/Normal-9442.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9442.png,Normal/masks/Normal-9442.png,True,False,299,299,L,112.23,ee00910fdae48957369e8c086d22ff7f
Normal-9443.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9443.png,Normal/images/Normal-9443.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9443.png,Normal/masks/Normal-9443.png,True,False,299,299,L,156.77,a2741ac0db8d25c57e759dea17d3dfd7
Normal-9444.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9444.png,Normal/images/Normal-9444.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9444.png,Normal/masks/Normal-9444.png,True,False,299,299,L,137.73,4a5d7d2a03dcb8bfb71ccfa236afc095
Normal-9445.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9445.png,Normal/images/Normal-9445.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9445.png,Normal/masks/Normal-9445.png,True,False,299,299,L,170.19,a80fadc465c2c649dd7e71edaeac4f43
Normal-9446.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9446.png,Normal/images/Normal-9446.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9446.png,Normal/masks/Normal-9446.png,True,False,299,299,L,126.57,e5b266d0d18c4d06192bbc3ff3a1f6d4
Normal-9447.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9447.png,Normal/images/Normal-9447.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9447.png,Normal/masks/Normal-9447.png,True,False,299,299,L,164.97,ec5bde3582b2dffccecd0426b8aeaa5a
Normal-9448.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9448.png,Normal/images/Normal-9448.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9448.png,Normal/masks/Normal-9448.png,True,False,299,299,L,128.28,18b80265d50c58d5c0ba79210d7334aa
Normal-9449.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9449.png,Normal/images/Normal-9449.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9449.png,Normal/masks/Normal-9449.png,True,False,299,299,L,154.09,de43501989aae1301fc5212ead9457af
Normal-945.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-945.png,Normal/images/Normal-945.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-945.png,Normal/masks/Normal-945.png,True,False,299,299,L,113.86,3fe5038406b0fea0ca5a6fed6cf3ce14
Normal-9450.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9450.png,Normal/images/Normal-9450.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9450.png,Normal/masks/Normal-9450.png,True,False,299,299,L,135.75,06491008e10327165bc66c904209d09a
Normal-9451.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9451.png,Normal/images/Normal-9451.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9451.png,Normal/masks/Normal-9451.png,True,False,299,299,L,129.99,8203a6c67aa32fceaa40a3ec8f0db6b0
Normal-9452.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9452.png,Normal/images/Normal-9452.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9452.png,Normal/masks/Normal-9452.png,True,False,299,299,L,62.91,d841d472b9316430d1ca65b762db80a7
Normal-9453.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9453.png,Normal/images/Normal-9453.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9453.png,Normal/masks/Normal-9453.png,True,False,299,299,L,132.58,117a97c1eb739ffbc13926990d0fa817
Normal-9454.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9454.png,Normal/images/Normal-9454.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9454.png,Normal/masks/Normal-9454.png,True,False,299,299,L,115.18,89b5ee1a28789a45958e4ef71ca44952
Normal-9455.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9455.png,Normal/images/Normal-9455.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9455.png,Normal/masks/Normal-9455.png,True,False,299,299,L,133.16,d2deac74479a2dcacf5492a8fdff0358
Normal-9456.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9456.png,Normal/images/Normal-9456.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9456.png,Normal/masks/Normal-9456.png,True,False,299,299,L,122.18,0531bc19cef9c3d4c7c3611b30cf1b2c
Normal-9457.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9457.png,Normal/images/Normal-9457.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9457.png,Normal/masks/Normal-9457.png,True,False,299,299,L,108.68,dd23fc238ff998ad51e57752d1ce7cb2
Normal-9458.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9458.png,Normal/images/Normal-9458.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9458.png,Normal/masks/Normal-9458.png,True,False,299,299,L,111.69,ae9f17d7045b36a4e7b0ac7401c48664
Normal-9459.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9459.png,Normal/images/Normal-9459.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9459.png,Normal/masks/Normal-9459.png,True,False,299,299,L,104.26,21e77948b26f8fd60fb0334212e9a4b0
Normal-946.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-946.png,Normal/images/Normal-946.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-946.png,Normal/masks/Normal-946.png,True,False,299,299,L,107.6,9b7778ba334372ee6399340049440bd9
Normal-9460.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9460.png,Normal/images/Normal-9460.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9460.png,Normal/masks/Normal-9460.png,True,False,299,299,L,154.62,e538456e43545f867420cc4bf1ec0c96
Normal-9461.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9461.png,Normal/images/Normal-9461.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9461.png,Normal/masks/Normal-9461.png,True,False,299,299,L,128.64,f2137da7094fea975e0cd821c71ed420
Normal-9462.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9462.png,Normal/images/Normal-9462.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9462.png,Normal/masks/Normal-9462.png,True,False,299,299,L,117.06,39796be3a3e91bb4a80fa9ef88ad90ff
Normal-9463.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9463.png,Normal/images/Normal-9463.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9463.png,Normal/masks/Normal-9463.png,True,False,299,299,L,133.5,6927caad2ffdaceaf904d851a1a73a74
Normal-9464.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9464.png,Normal/images/Normal-9464.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9464.png,Normal/masks/Normal-9464.png,True,False,299,299,L,167.58,9ec55b63fac620d05d6d659e5c27b33a
Normal-9465.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9465.png,Normal/images/Normal-9465.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9465.png,Normal/masks/Normal-9465.png,True,False,299,299,L,103.61,c652f1370eadc5329ad07bf09551fb0e
Normal-9466.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9466.png,Normal/images/Normal-9466.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9466.png,Normal/masks/Normal-9466.png,True,False,299,299,L,136.84,5f48e1d56e79afa15bde77bc56f8ac9b
Normal-9467.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9467.png,Normal/images/Normal-9467.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9467.png,Normal/masks/Normal-9467.png,True,False,299,299,L,108.94,9bb599682829a9a931fc08e145df55fb
Normal-9468.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9468.png,Normal/images/Normal-9468.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9468.png,Normal/masks/Normal-9468.png,True,False,299,299,L,111.56,86e86ab7c9828bfcd4ca6414e85c6a93
Normal-9469.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9469.png,Normal/images/Normal-9469.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9469.png,Normal/masks/Normal-9469.png,True,False,299,299,L,114.21,1e4092115d3d88b97484fcfda6ef4fd0
Normal-947.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-947.png,Normal/images/Normal-947.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-947.png,Normal/masks/Normal-947.png,True,False,299,299,L,114.69,d0ce181b5f87bda63103181d0c653e59
Normal-9470.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9470.png,Normal/images/Normal-9470.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9470.png,Normal/masks/Normal-9470.png,True,False,299,299,L,103.54,99effa61daf40a1ff6a9b566c1d46e10
Normal-9471.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9471.png,Normal/images/Normal-9471.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9471.png,Normal/masks/Normal-9471.png,True,False,299,299,L,147.02,bab557e68f33de78fd9ea365d1c488fd
Normal-9472.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9472.png,Normal/images/Normal-9472.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9472.png,Normal/masks/Normal-9472.png,True,False,299,299,L,102.83,5bcdbbdd369304370806beca6e0a8c80
Normal-9473.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9473.png,Normal/images/Normal-9473.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9473.png,Normal/masks/Normal-9473.png,True,False,299,299,L,129.73,c26f37c8ce46c7e4c269c057728645b7
Normal-9474.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9474.png,Normal/images/Normal-9474.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9474.png,Normal/masks/Normal-9474.png,True,False,299,299,L,125.19,a9f964f477c5cb33fb6f795185b9d7ef
Normal-9475.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9475.png,Normal/images/Normal-9475.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9475.png,Normal/masks/Normal-9475.png,True,False,299,299,L,155.37,e2ea5dffe3f6ce07bc418ca87b4dd6f2
Normal-9476.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9476.png,Normal/images/Normal-9476.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9476.png,Normal/masks/Normal-9476.png,True,False,299,299,L,114.71,cf726902eb8ac2e35714789a3237a0e7
Normal-9477.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9477.png,Normal/images/Normal-9477.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9477.png,Normal/masks/Normal-9477.png,True,False,299,299,L,164.35,adafd06a856dfdf58ebf9bf76bad137e
Normal-9478.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9478.png,Normal/images/Normal-9478.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9478.png,Normal/masks/Normal-9478.png,True,False,299,299,L,120.52,f775de95a5f81061fc206d8c9e5cf810
Normal-9479.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9479.png,Normal/images/Normal-9479.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9479.png,Normal/masks/Normal-9479.png,True,False,299,299,L,165.03,dd92889e13bacac504505d8ee269cee6
Normal-948.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-948.png,Normal/images/Normal-948.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-948.png,Normal/masks/Normal-948.png,True,False,299,299,L,118.35,93b045b9dd850436f60547f2040d2caa
Normal-9480.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9480.png,Normal/images/Normal-9480.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9480.png,Normal/masks/Normal-9480.png,True,False,299,299,L,149.42,1d923e5d3555226cbd48f8e5d94f6611
Normal-9481.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9481.png,Normal/images/Normal-9481.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9481.png,Normal/masks/Normal-9481.png,True,False,299,299,L,111.56,614036e43a4627ddd6474ca62772574b
Normal-9482.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9482.png,Normal/images/Normal-9482.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9482.png,Normal/masks/Normal-9482.png,True,False,299,299,L,93.16,c2bd622ba5c7633474e731766c1880b7
Normal-9483.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9483.png,Normal/images/Normal-9483.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9483.png,Normal/masks/Normal-9483.png,True,False,299,299,L,111.23,b8f30fad88e567121d30f547da9d3b3a
Normal-9484.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9484.png,Normal/images/Normal-9484.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9484.png,Normal/masks/Normal-9484.png,True,False,299,299,L,99.58,9b5eec0ad7b5ace57b11b94a48044d65
Normal-9485.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9485.png,Normal/images/Normal-9485.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9485.png,Normal/masks/Normal-9485.png,True,False,299,299,L,115.86,6b6940fa0d4a623aaa18594ff531b38f
Normal-9486.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9486.png,Normal/images/Normal-9486.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9486.png,Normal/masks/Normal-9486.png,True,False,299,299,L,116.09,563820543ef67caab22f94eb20129b29
Normal-9487.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9487.png,Normal/images/Normal-9487.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9487.png,Normal/masks/Normal-9487.png,True,False,299,299,L,104.89,534d8df243cc608f72db63b8990d503d
Normal-9488.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9488.png,Normal/images/Normal-9488.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9488.png,Normal/masks/Normal-9488.png,True,False,299,299,L,138.6,1dbdffde7b6b117a17b475f377a5d71a
Normal-9489.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9489.png,Normal/images/Normal-9489.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9489.png,Normal/masks/Normal-9489.png,True,False,299,299,L,112.06,03c4f3d4a31a8a0b2f5543f3a2f78263
Normal-949.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-949.png,Normal/images/Normal-949.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-949.png,Normal/masks/Normal-949.png,True,False,299,299,L,119.04,d8c90419917da7f4bf841b6faad882ed
Normal-9490.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9490.png,Normal/images/Normal-9490.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9490.png,Normal/masks/Normal-9490.png,True,False,299,299,L,127.57,98f8fcef02255501a42262bbe14e75ce
Normal-9491.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9491.png,Normal/images/Normal-9491.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9491.png,Normal/masks/Normal-9491.png,True,False,299,299,L,175.76,1b28d2ebbd772aae09ce5c209cfefd88
Normal-9492.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9492.png,Normal/images/Normal-9492.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9492.png,Normal/masks/Normal-9492.png,True,False,299,299,L,104.89,32fa9191f1e15435f7839fbf6a145305
Normal-9493.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9493.png,Normal/images/Normal-9493.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9493.png,Normal/masks/Normal-9493.png,True,False,299,299,L,122.25,10e981671990f741406cf388a1756d14
Normal-9494.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9494.png,Normal/images/Normal-9494.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9494.png,Normal/masks/Normal-9494.png,True,False,299,299,L,122.06,ed6ec4caa6178ef2824c9b267305e953
Normal-9495.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9495.png,Normal/images/Normal-9495.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9495.png,Normal/masks/Normal-9495.png,True,False,299,299,L,168.73,7dc187003f2f478f2f4466565ef52247
Normal-9496.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9496.png,Normal/images/Normal-9496.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9496.png,Normal/masks/Normal-9496.png,True,False,299,299,L,141.55,c7c27f467e7197352987f40f703e3a11
Normal-9497.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9497.png,Normal/images/Normal-9497.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9497.png,Normal/masks/Normal-9497.png,True,False,299,299,L,108.55,1fb1094579d90b06979c7dd62cca8ae8
Normal-9498.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9498.png,Normal/images/Normal-9498.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9498.png,Normal/masks/Normal-9498.png,True,False,299,299,L,129.67,c546bdc5d32991ad9ad3ee1b4925fe66
Normal-9499.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9499.png,Normal/images/Normal-9499.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9499.png,Normal/masks/Normal-9499.png,True,False,299,299,L,114.73,d3c9ddd2bcb23ef2aa28f373c923903d
Normal-95.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-95.png,Normal/images/Normal-95.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-95.png,Normal/masks/Normal-95.png,True,False,299,299,L,142.17,2b824dbd1798cbbd8c612462f324d0d4
Normal-950.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-950.png,Normal/images/Normal-950.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-950.png,Normal/masks/Normal-950.png,True,False,299,299,L,102.29,2afac4f2c080a5114cd5f292d07d50bf
Normal-9500.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9500.png,Normal/images/Normal-9500.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9500.png,Normal/masks/Normal-9500.png,True,False,299,299,L,145.72,d3009e43f100eb62624ea496632df41e
Normal-9501.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9501.png,Normal/images/Normal-9501.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9501.png,Normal/masks/Normal-9501.png,True,False,299,299,L,105.77,3ac6d3fbee504538b5feaed941ffe487
Normal-9502.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9502.png,Normal/images/Normal-9502.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9502.png,Normal/masks/Normal-9502.png,True,False,299,299,L,153.64,99a64a9ad71086411589af75321e02f7
Normal-9503.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9503.png,Normal/images/Normal-9503.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9503.png,Normal/masks/Normal-9503.png,True,False,299,299,L,113.15,c307c3545140295f53c7f2edd8c7797b
Normal-9504.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9504.png,Normal/images/Normal-9504.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9504.png,Normal/masks/Normal-9504.png,True,False,299,299,L,120.63,958569f6188b5d9953df5dde477283b7
Normal-9505.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9505.png,Normal/images/Normal-9505.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9505.png,Normal/masks/Normal-9505.png,True,False,299,299,L,136.61,ddef46d91e192b77d5113d19c56a2121
Normal-9506.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9506.png,Normal/images/Normal-9506.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9506.png,Normal/masks/Normal-9506.png,True,False,299,299,L,105.98,40181736ada3dfe115957c7e79959228
Normal-9507.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9507.png,Normal/images/Normal-9507.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9507.png,Normal/masks/Normal-9507.png,True,False,299,299,L,115.58,6d69e30d6a099c454a29cb666d322d14
Normal-9508.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9508.png,Normal/images/Normal-9508.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9508.png,Normal/masks/Normal-9508.png,True,False,299,299,L,167.63,ce0b071b307d5a323eff09ec8fe6e9b0
Normal-9509.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9509.png,Normal/images/Normal-9509.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9509.png,Normal/masks/Normal-9509.png,True,False,299,299,L,111.74,a4f1a11b46dcde454ae6b0e95411e932
Normal-951.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-951.png,Normal/images/Normal-951.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-951.png,Normal/masks/Normal-951.png,True,False,299,299,L,140.8,be09901e6a1822787f75444f440e7e15
Normal-9510.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9510.png,Normal/images/Normal-9510.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9510.png,Normal/masks/Normal-9510.png,True,False,299,299,L,150.21,7d9493c075257cf119942a44e8324453
Normal-9511.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9511.png,Normal/images/Normal-9511.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9511.png,Normal/masks/Normal-9511.png,True,False,299,299,L,162.83,5eb14dd06652a5f3e5228fd2eb1f7360
Normal-9512.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9512.png,Normal/images/Normal-9512.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9512.png,Normal/masks/Normal-9512.png,True,False,299,299,L,120.21,c71a9a436ef9e35e5bce3d3a93d57470
Normal-9513.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9513.png,Normal/images/Normal-9513.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9513.png,Normal/masks/Normal-9513.png,True,False,299,299,L,107.03,35bad76af5f94199197ede9ee9a4e595
Normal-9514.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9514.png,Normal/images/Normal-9514.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9514.png,Normal/masks/Normal-9514.png,True,False,299,299,L,111.26,3cdfa51880219cfa5cdf7d6804257d54
Normal-9515.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9515.png,Normal/images/Normal-9515.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9515.png,Normal/masks/Normal-9515.png,True,False,299,299,L,120.1,6d1f350f5be721b5912a0117209eef32
Normal-9516.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9516.png,Normal/images/Normal-9516.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9516.png,Normal/masks/Normal-9516.png,True,False,299,299,L,138.9,4465992b4502e632883fd4ace0c56ec8
Normal-9517.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9517.png,Normal/images/Normal-9517.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9517.png,Normal/masks/Normal-9517.png,True,False,299,299,L,121.36,f62a92c72c53d38c6c764bf0d001032a
Normal-9518.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9518.png,Normal/images/Normal-9518.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9518.png,Normal/masks/Normal-9518.png,True,False,299,299,L,133.35,bb64edb2304d1cfa3d8a6510e9dda7c0
Normal-9519.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9519.png,Normal/images/Normal-9519.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9519.png,Normal/masks/Normal-9519.png,True,False,299,299,L,101.63,2eda42ee28eb61985888e725e81a0a22
Normal-952.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-952.png,Normal/images/Normal-952.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-952.png,Normal/masks/Normal-952.png,True,False,299,299,L,129.46,b358d857be4ce2d1c1cea6ac311e454d
Normal-9520.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9520.png,Normal/images/Normal-9520.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9520.png,Normal/masks/Normal-9520.png,True,False,299,299,L,178.61,3caceea77dbad427f5ae4d3d0e7d5318
Normal-9521.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9521.png,Normal/images/Normal-9521.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9521.png,Normal/masks/Normal-9521.png,True,False,299,299,L,107.82,adc901e3e623f351781177f3702e6b1e
Normal-9522.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9522.png,Normal/images/Normal-9522.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9522.png,Normal/masks/Normal-9522.png,True,False,299,299,L,125.91,dbcd7e9b2317257e8c0ddf55bc97f3e7
Normal-9523.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9523.png,Normal/images/Normal-9523.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9523.png,Normal/masks/Normal-9523.png,True,False,299,299,L,138.11,7597301e80217db4eec9e72259f0d325
Normal-9524.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9524.png,Normal/images/Normal-9524.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9524.png,Normal/masks/Normal-9524.png,True,False,299,299,L,158.34,90d930bd162e2289a4b1d98285cdf92c
Normal-9525.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9525.png,Normal/images/Normal-9525.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9525.png,Normal/masks/Normal-9525.png,True,False,299,299,L,144.79,1ee57e532c3165610b584e3cbbfe1348
Normal-9526.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9526.png,Normal/images/Normal-9526.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9526.png,Normal/masks/Normal-9526.png,True,False,299,299,L,170.7,93732ecc52ae7b84f5fc5777959921f2
Normal-9527.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9527.png,Normal/images/Normal-9527.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9527.png,Normal/masks/Normal-9527.png,True,False,299,299,L,171.41,7d90c73ce26b8f2d3faa41e346cce259
Normal-9528.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9528.png,Normal/images/Normal-9528.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9528.png,Normal/masks/Normal-9528.png,True,False,299,299,L,113.8,10674407bdf51542a8c41d0e27b5f999
Normal-9529.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9529.png,Normal/images/Normal-9529.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9529.png,Normal/masks/Normal-9529.png,True,False,299,299,L,137.65,7f41a3212a522ad1325700faa6ee98d1
Normal-953.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-953.png,Normal/images/Normal-953.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-953.png,Normal/masks/Normal-953.png,True,False,299,299,L,146.01,b49ce2b40aeaff11763488ba6c506729
Normal-9530.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9530.png,Normal/images/Normal-9530.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9530.png,Normal/masks/Normal-9530.png,True,False,299,299,L,123.68,b245479ff5edf3950414adf7f8a1553b
Normal-9531.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9531.png,Normal/images/Normal-9531.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9531.png,Normal/masks/Normal-9531.png,True,False,299,299,L,154.04,96d7152a523e329dfba2ce41b4cc448e
Normal-9532.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9532.png,Normal/images/Normal-9532.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9532.png,Normal/masks/Normal-9532.png,True,False,299,299,L,158.46,7cd2a30277ed3c58987c5d17fe7a5e26
Normal-9533.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9533.png,Normal/images/Normal-9533.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9533.png,Normal/masks/Normal-9533.png,True,False,299,299,L,124.41,e5ecb2560221148dd87f2ef9f4084c25
Normal-9534.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9534.png,Normal/images/Normal-9534.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9534.png,Normal/masks/Normal-9534.png,True,False,299,299,L,103.36,7183bc3949eeda8e90b6419aeba30eee
Normal-9535.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9535.png,Normal/images/Normal-9535.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9535.png,Normal/masks/Normal-9535.png,True,False,299,299,L,104.67,907c3a8ff7e04a40aec5a1605e3de392
Normal-9536.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9536.png,Normal/images/Normal-9536.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9536.png,Normal/masks/Normal-9536.png,True,False,299,299,L,106.56,293749e85c9a04e896a3dd3e1a3b06f4
Normal-9537.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9537.png,Normal/images/Normal-9537.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9537.png,Normal/masks/Normal-9537.png,True,False,299,299,L,160.61,cd445fde1ad563163200ca8c00b0283a
Normal-9538.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9538.png,Normal/images/Normal-9538.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9538.png,Normal/masks/Normal-9538.png,True,False,299,299,L,137.53,fc18bd1d69481ee759a15bbc19a5f4c1
Normal-9539.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9539.png,Normal/images/Normal-9539.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9539.png,Normal/masks/Normal-9539.png,True,False,299,299,L,106.64,41aaf168c9f5ac0314fff84ad6a99e0f
Normal-954.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-954.png,Normal/images/Normal-954.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-954.png,Normal/masks/Normal-954.png,True,False,299,299,L,123.86,f93a01d2fd8931f355399087c46a5d77
Normal-9540.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9540.png,Normal/images/Normal-9540.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9540.png,Normal/masks/Normal-9540.png,True,False,299,299,L,110.94,be57e7ca23672ef5b5a55396377b989e
Normal-9541.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9541.png,Normal/images/Normal-9541.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9541.png,Normal/masks/Normal-9541.png,True,False,299,299,L,114.28,ef92ff9fef143036af11a3568c7b4620
Normal-9542.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9542.png,Normal/images/Normal-9542.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9542.png,Normal/masks/Normal-9542.png,True,False,299,299,L,109.8,0036f50ccfaf7e07031407153edaee12
Normal-9543.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9543.png,Normal/images/Normal-9543.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9543.png,Normal/masks/Normal-9543.png,True,False,299,299,L,123.05,ddaafc184c3d585d4b64dd8fa559f985
Normal-9544.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9544.png,Normal/images/Normal-9544.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9544.png,Normal/masks/Normal-9544.png,True,False,299,299,L,124.85,487e2ffedb3dca707f3df201c2a6ebff
Normal-9545.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9545.png,Normal/images/Normal-9545.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9545.png,Normal/masks/Normal-9545.png,True,False,299,299,L,131.9,a99e7200a36a18228cd3ae0fd66e1795
Normal-9546.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9546.png,Normal/images/Normal-9546.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9546.png,Normal/masks/Normal-9546.png,True,False,299,299,L,116.51,085b99ed832182393983b4c045c7fd80
Normal-9547.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9547.png,Normal/images/Normal-9547.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9547.png,Normal/masks/Normal-9547.png,True,False,299,299,L,99.47,37015f72527a2915f71c0cdde28b8a61
Normal-9548.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9548.png,Normal/images/Normal-9548.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9548.png,Normal/masks/Normal-9548.png,True,False,299,299,L,152.39,c32da4e92eada4c50e9bbda29604d064
Normal-9549.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9549.png,Normal/images/Normal-9549.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9549.png,Normal/masks/Normal-9549.png,True,False,299,299,L,119.92,b4fdec8cea5ca36d7a32da53cc909a49
Normal-955.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-955.png,Normal/images/Normal-955.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-955.png,Normal/masks/Normal-955.png,True,False,299,299,L,136.86,905cf269a5030e699590ec33b2a4b210
Normal-9550.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9550.png,Normal/images/Normal-9550.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9550.png,Normal/masks/Normal-9550.png,True,False,299,299,L,136.52,6919223117270239143bc39c05ff68e9
Normal-9551.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9551.png,Normal/images/Normal-9551.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9551.png,Normal/masks/Normal-9551.png,True,False,299,299,L,125.07,d334e81164034aecd2424b43ab378bda
Normal-9552.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9552.png,Normal/images/Normal-9552.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9552.png,Normal/masks/Normal-9552.png,True,False,299,299,L,152.38,2b168b731dc053107bec8318372b2889
Normal-9553.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9553.png,Normal/images/Normal-9553.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9553.png,Normal/masks/Normal-9553.png,True,False,299,299,L,123.71,f660accff4a22e6222e2697309d6e884
Normal-9554.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9554.png,Normal/images/Normal-9554.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9554.png,Normal/masks/Normal-9554.png,True,False,299,299,L,125.56,af26d5d7b196e70d7b95d90e70bb5623
Normal-9555.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9555.png,Normal/images/Normal-9555.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9555.png,Normal/masks/Normal-9555.png,True,False,299,299,L,110.83,d9fcce4af466e9c083a7f367ea7cd55d
Normal-9556.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9556.png,Normal/images/Normal-9556.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9556.png,Normal/masks/Normal-9556.png,True,False,299,299,L,166.36,c5477d4426c607f4d8297465bf54be6e
Normal-9557.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9557.png,Normal/images/Normal-9557.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9557.png,Normal/masks/Normal-9557.png,True,False,299,299,L,114.37,2122f1783d04745a57696f0835383d55
Normal-9558.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9558.png,Normal/images/Normal-9558.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9558.png,Normal/masks/Normal-9558.png,True,False,299,299,L,127.48,1d99f93a789d3431cb7c39cd8af4a4e6
Normal-9559.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9559.png,Normal/images/Normal-9559.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9559.png,Normal/masks/Normal-9559.png,True,False,299,299,L,165.33,5b87f72b332353a6253957db01b117dd
Normal-956.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-956.png,Normal/images/Normal-956.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-956.png,Normal/masks/Normal-956.png,True,False,299,299,L,116.99,22796a87818b63d3e7415eed3a4747c9
Normal-9560.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9560.png,Normal/images/Normal-9560.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9560.png,Normal/masks/Normal-9560.png,True,False,299,299,L,117.02,6320b54939d7e4a5d01f9eb7c9695bcd
Normal-9561.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9561.png,Normal/images/Normal-9561.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9561.png,Normal/masks/Normal-9561.png,True,False,299,299,L,159.8,0d991259e2d44aa61e8fa089d55d4d77
Normal-9562.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9562.png,Normal/images/Normal-9562.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9562.png,Normal/masks/Normal-9562.png,True,False,299,299,L,134.12,61dff20e9affd61d1f8c45ebbe2abcd0
Normal-9563.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9563.png,Normal/images/Normal-9563.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9563.png,Normal/masks/Normal-9563.png,True,False,299,299,L,153.97,d6389e820acca72fd1e2ca8ad148dbcc
Normal-9564.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9564.png,Normal/images/Normal-9564.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9564.png,Normal/masks/Normal-9564.png,True,False,299,299,L,162.37,7b704925063a738e44b84d5b15634dbb
Normal-9565.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9565.png,Normal/images/Normal-9565.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9565.png,Normal/masks/Normal-9565.png,True,False,299,299,L,130.75,4d2e6e16958fbd95a2a80fe42a247cd0
Normal-9566.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9566.png,Normal/images/Normal-9566.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9566.png,Normal/masks/Normal-9566.png,True,False,299,299,L,185.35,0a11c2a526ab814739aa314588482386
Normal-9567.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9567.png,Normal/images/Normal-9567.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9567.png,Normal/masks/Normal-9567.png,True,False,299,299,L,115.62,e8d518814d7ab58a84b996ff5811fa5b
Normal-9568.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9568.png,Normal/images/Normal-9568.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9568.png,Normal/masks/Normal-9568.png,True,False,299,299,L,135.42,3f85f9a0e31413fc80b366c0352a68a0
Normal-9569.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9569.png,Normal/images/Normal-9569.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9569.png,Normal/masks/Normal-9569.png,True,False,299,299,L,114.44,348411d125a1441ef1855702753152dc
Normal-957.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-957.png,Normal/images/Normal-957.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-957.png,Normal/masks/Normal-957.png,True,False,299,299,L,123.09,8a2187d023f117f34205a64921dd3b24
Normal-9570.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9570.png,Normal/images/Normal-9570.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9570.png,Normal/masks/Normal-9570.png,True,False,299,299,L,176.57,442f18291ddbbf65ffb4e2c32447e1f1
Normal-9571.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9571.png,Normal/images/Normal-9571.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9571.png,Normal/masks/Normal-9571.png,True,False,299,299,L,155.62,d0970cc805ab6e3b758809d05d44d18d
Normal-9572.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9572.png,Normal/images/Normal-9572.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9572.png,Normal/masks/Normal-9572.png,True,False,299,299,L,118.6,b1616bae52503ff20be7bbb8d91eb88d
Normal-9573.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9573.png,Normal/images/Normal-9573.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9573.png,Normal/masks/Normal-9573.png,True,False,299,299,L,121.9,c8334c5da08d17fc8f88568be094e10b
Normal-9574.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9574.png,Normal/images/Normal-9574.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9574.png,Normal/masks/Normal-9574.png,True,False,299,299,L,117.79,2474ddbbd71d79c1c1ec8b85a23de442
Normal-9575.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9575.png,Normal/images/Normal-9575.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9575.png,Normal/masks/Normal-9575.png,True,False,299,299,L,120.97,e73f48240afa6b2c91e6c20de480d145
Normal-9576.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9576.png,Normal/images/Normal-9576.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9576.png,Normal/masks/Normal-9576.png,True,False,299,299,L,113.49,860242beb5742a83181d5b53d30a5507
Normal-9577.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9577.png,Normal/images/Normal-9577.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9577.png,Normal/masks/Normal-9577.png,True,False,299,299,L,164.14,a98edd07c2e004454b27dbe3cbe4d751
Normal-9578.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9578.png,Normal/images/Normal-9578.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9578.png,Normal/masks/Normal-9578.png,True,False,299,299,L,117.67,cb34533043216c0c504fde2c71ec1104
Normal-9579.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9579.png,Normal/images/Normal-9579.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9579.png,Normal/masks/Normal-9579.png,True,False,299,299,L,139.35,fa43d53d04b8edf1e2076e60730597da
Normal-958.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-958.png,Normal/images/Normal-958.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-958.png,Normal/masks/Normal-958.png,True,False,299,299,L,113.67,b9ebad3aec29cb1894d7d50ed5394f56
Normal-9580.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9580.png,Normal/images/Normal-9580.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9580.png,Normal/masks/Normal-9580.png,True,False,299,299,L,171.6,b4ab93ca62265c6213666e6d657d5ece
Normal-9581.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9581.png,Normal/images/Normal-9581.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9581.png,Normal/masks/Normal-9581.png,True,False,299,299,L,115.08,b0a3db9be2d885b9dddb6cd0e5915e0e
Normal-9582.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9582.png,Normal/images/Normal-9582.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9582.png,Normal/masks/Normal-9582.png,True,False,299,299,L,131.82,ab000af2d65c2603f49a31f651df6998
Normal-9583.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9583.png,Normal/images/Normal-9583.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9583.png,Normal/masks/Normal-9583.png,True,False,299,299,L,162.9,e3e46c53de74d029847d2f06c3f83fe6
Normal-9584.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9584.png,Normal/images/Normal-9584.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9584.png,Normal/masks/Normal-9584.png,True,False,299,299,L,110.6,cc563abfe2383a6090adcdc2935daecb
Normal-9585.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9585.png,Normal/images/Normal-9585.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9585.png,Normal/masks/Normal-9585.png,True,False,299,299,L,167.04,558a59fe923d95a217a541a884a2ab10
Normal-9586.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9586.png,Normal/images/Normal-9586.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9586.png,Normal/masks/Normal-9586.png,True,False,299,299,L,140.51,65190368157a13bcc03893a07e44adde
Normal-9587.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9587.png,Normal/images/Normal-9587.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9587.png,Normal/masks/Normal-9587.png,True,False,299,299,L,127.03,11a41b8128448631da1e02247926a281
Normal-9588.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9588.png,Normal/images/Normal-9588.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9588.png,Normal/masks/Normal-9588.png,True,False,299,299,L,158.0,47ab86a1987ca98185ca716146729c27
Normal-9589.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9589.png,Normal/images/Normal-9589.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9589.png,Normal/masks/Normal-9589.png,True,False,299,299,L,128.87,c37a8d643aac7f9a58dbda75f956753d
Normal-959.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-959.png,Normal/images/Normal-959.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-959.png,Normal/masks/Normal-959.png,True,False,299,299,L,113.86,c9797ad7aac24305deea53d49dc23264
Normal-9590.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9590.png,Normal/images/Normal-9590.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9590.png,Normal/masks/Normal-9590.png,True,False,299,299,L,143.63,a4f67cf73e356560fcda587805d64d15
Normal-9591.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9591.png,Normal/images/Normal-9591.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9591.png,Normal/masks/Normal-9591.png,True,False,299,299,L,47.25,dae6471d232861bf9b936fcb21791ca0
Normal-9592.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9592.png,Normal/images/Normal-9592.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9592.png,Normal/masks/Normal-9592.png,True,False,299,299,L,114.13,e944f3d88907427327ae98c7af8bb9b0
Normal-9593.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9593.png,Normal/images/Normal-9593.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9593.png,Normal/masks/Normal-9593.png,True,False,299,299,L,106.44,e3602ccfc2fcb0048c42d812b7f5018f
Normal-9594.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9594.png,Normal/images/Normal-9594.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9594.png,Normal/masks/Normal-9594.png,True,False,299,299,L,109.26,7f7a72b84df838ce47b9446c11154a4c
Normal-9595.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9595.png,Normal/images/Normal-9595.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9595.png,Normal/masks/Normal-9595.png,True,False,299,299,L,134.05,d1daf4ce2d3c15b27b2fe2ac0fdd7b60
Normal-9596.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9596.png,Normal/images/Normal-9596.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9596.png,Normal/masks/Normal-9596.png,True,False,299,299,L,165.99,36920e9cb8e4c4f044ed2e8c3f061de9
Normal-9597.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9597.png,Normal/images/Normal-9597.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9597.png,Normal/masks/Normal-9597.png,True,False,299,299,L,118.54,a6b5cdb818b76b5a731be1b9ff16435c
Normal-9598.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9598.png,Normal/images/Normal-9598.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9598.png,Normal/masks/Normal-9598.png,True,False,299,299,L,177.6,525460f3ab951d9ef5834540742960f6
Normal-9599.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9599.png,Normal/images/Normal-9599.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9599.png,Normal/masks/Normal-9599.png,True,False,299,299,L,154.53,fbe05c08b2e424e1a4f674d23ed5aed4
Normal-96.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-96.png,Normal/images/Normal-96.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-96.png,Normal/masks/Normal-96.png,True,False,299,299,L,138.4,60894cea1d32bd5fd90a94b28febe338
Normal-960.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-960.png,Normal/images/Normal-960.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-960.png,Normal/masks/Normal-960.png,True,False,299,299,L,120.81,746df0382b43cffb14547f039a775661
Normal-9600.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9600.png,Normal/images/Normal-9600.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9600.png,Normal/masks/Normal-9600.png,True,False,299,299,L,165.39,c7a5ceb0e63e36ef1495cf07c5826121
Normal-9601.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9601.png,Normal/images/Normal-9601.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9601.png,Normal/masks/Normal-9601.png,True,False,299,299,L,126.0,b69192a0adef9e6472fb3b638b811b2f
Normal-9602.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9602.png,Normal/images/Normal-9602.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9602.png,Normal/masks/Normal-9602.png,True,False,299,299,L,110.5,48046f0449a30020684a997fa8d2845c
Normal-9603.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9603.png,Normal/images/Normal-9603.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9603.png,Normal/masks/Normal-9603.png,True,False,299,299,L,118.57,d735c91f02963ebc61027a143e1b0bc1
Normal-9604.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9604.png,Normal/images/Normal-9604.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9604.png,Normal/masks/Normal-9604.png,True,False,299,299,L,146.61,34be23780c29e99672afcdbebb4830b0
Normal-9605.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9605.png,Normal/images/Normal-9605.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9605.png,Normal/masks/Normal-9605.png,True,False,299,299,L,136.02,a9c02bea22da78b1683126e020ab55b0
Normal-9606.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9606.png,Normal/images/Normal-9606.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9606.png,Normal/masks/Normal-9606.png,True,False,299,299,L,118.09,e752ec73603437d9aae2c632a61cef5d
Normal-9607.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9607.png,Normal/images/Normal-9607.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9607.png,Normal/masks/Normal-9607.png,True,False,299,299,L,150.29,0e137529416d0b7cce67b4597c956765
Normal-9608.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9608.png,Normal/images/Normal-9608.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9608.png,Normal/masks/Normal-9608.png,True,False,299,299,L,121.77,a9499af4c54a2f066b4f472cc571cafd
Normal-9609.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9609.png,Normal/images/Normal-9609.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9609.png,Normal/masks/Normal-9609.png,True,False,299,299,L,134.51,c6b395a74eeb8ce8c5ba055de28ed5ba
Normal-961.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-961.png,Normal/images/Normal-961.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-961.png,Normal/masks/Normal-961.png,True,False,299,299,L,128.01,9b9cda6879273448eb3bbb08e79beaf2
Normal-9610.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9610.png,Normal/images/Normal-9610.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9610.png,Normal/masks/Normal-9610.png,True,False,299,299,L,119.65,d68b83e3278d413d1a912f2dc2cc0f2f
Normal-9611.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9611.png,Normal/images/Normal-9611.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9611.png,Normal/masks/Normal-9611.png,True,False,299,299,L,179.99,9ab7447af9303f67f05b59aac110e823
Normal-9612.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9612.png,Normal/images/Normal-9612.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9612.png,Normal/masks/Normal-9612.png,True,False,299,299,L,111.72,cf75605b4a9a20f704f011537fc03c7e
Normal-9613.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9613.png,Normal/images/Normal-9613.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9613.png,Normal/masks/Normal-9613.png,True,False,299,299,L,130.25,3e7f98e463a9f1dbb333e8758ebea72c
Normal-9614.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9614.png,Normal/images/Normal-9614.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9614.png,Normal/masks/Normal-9614.png,True,False,299,299,L,163.16,fe83585270584f92cbe5db0adf3564fd
Normal-9615.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9615.png,Normal/images/Normal-9615.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9615.png,Normal/masks/Normal-9615.png,True,False,299,299,L,127.65,a5030729e1bc7a30ed907dc3882a2985
Normal-9616.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9616.png,Normal/images/Normal-9616.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9616.png,Normal/masks/Normal-9616.png,True,False,299,299,L,137.65,3c14e508b4f8449bab3b34de415328cd
Normal-9617.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9617.png,Normal/images/Normal-9617.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9617.png,Normal/masks/Normal-9617.png,True,False,299,299,L,127.03,5d755428497380d2078265bbfb586a77
Normal-9618.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9618.png,Normal/images/Normal-9618.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9618.png,Normal/masks/Normal-9618.png,True,False,299,299,L,110.9,60b61fb126a1c38d2bb44ee4aaee6c21
Normal-9619.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9619.png,Normal/images/Normal-9619.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9619.png,Normal/masks/Normal-9619.png,True,False,299,299,L,165.82,f7b2a6fd28b720ff9d8938ae3daf45fe
Normal-962.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-962.png,Normal/images/Normal-962.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-962.png,Normal/masks/Normal-962.png,True,False,299,299,L,128.62,e0d0983ba81b53d71c27c40116f3871a
Normal-9620.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9620.png,Normal/images/Normal-9620.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9620.png,Normal/masks/Normal-9620.png,True,False,299,299,L,104.02,a79ffaf28e02395f50c1e527a6385bdc
Normal-9621.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9621.png,Normal/images/Normal-9621.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9621.png,Normal/masks/Normal-9621.png,True,False,299,299,L,151.54,bd914fc133a5ce2dff3396d703ffa19d
Normal-9622.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9622.png,Normal/images/Normal-9622.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9622.png,Normal/masks/Normal-9622.png,True,False,299,299,L,133.0,c15edffc4ce8337af5887d54886de403
Normal-9623.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9623.png,Normal/images/Normal-9623.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9623.png,Normal/masks/Normal-9623.png,True,False,299,299,L,164.87,c3e98ec8bebefcc97bb138087b465f46
Normal-9624.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9624.png,Normal/images/Normal-9624.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9624.png,Normal/masks/Normal-9624.png,True,False,299,299,L,133.78,5ebda8f67d177878ee5e5d48e5b372d3
Normal-9625.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9625.png,Normal/images/Normal-9625.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9625.png,Normal/masks/Normal-9625.png,True,False,299,299,L,113.14,69baa4e924d4f864a4efb67f2ca200cb
Normal-9626.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9626.png,Normal/images/Normal-9626.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9626.png,Normal/masks/Normal-9626.png,True,False,299,299,L,125.69,79fb29071efe01a3a81167e6a32231c5
Normal-9627.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9627.png,Normal/images/Normal-9627.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9627.png,Normal/masks/Normal-9627.png,True,False,299,299,L,84.68,79346434bd94f88bfbaa818df1afda8b
Normal-9628.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9628.png,Normal/images/Normal-9628.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9628.png,Normal/masks/Normal-9628.png,True,False,299,299,L,136.08,eb7bbb8bea877c64fca45fb523881d80
Normal-9629.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9629.png,Normal/images/Normal-9629.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9629.png,Normal/masks/Normal-9629.png,True,False,299,299,L,162.78,bc50b1c2e71b9e5b057b8b2b0cac562d
Normal-963.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-963.png,Normal/images/Normal-963.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-963.png,Normal/masks/Normal-963.png,True,False,299,299,L,121.35,14ec08251751646daf5f1f0d3207e42d
Normal-9630.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9630.png,Normal/images/Normal-9630.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9630.png,Normal/masks/Normal-9630.png,True,False,299,299,L,130.03,6ee365722e1aa26b324d884acabcdc0e
Normal-9631.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9631.png,Normal/images/Normal-9631.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9631.png,Normal/masks/Normal-9631.png,True,False,299,299,L,117.82,b3e233e1b59d0a8f1a62a02b43f2044c
Normal-9632.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9632.png,Normal/images/Normal-9632.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9632.png,Normal/masks/Normal-9632.png,True,False,299,299,L,115.88,b3bd88d8a63b171b167a0c333c32df3d
Normal-9633.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9633.png,Normal/images/Normal-9633.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9633.png,Normal/masks/Normal-9633.png,True,False,299,299,L,155.84,1bbe869c1d836ff9586ad33007cb204c
Normal-9634.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9634.png,Normal/images/Normal-9634.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9634.png,Normal/masks/Normal-9634.png,True,False,299,299,L,110.64,4a1c736d30c5b00c600d550239703a0e
Normal-9635.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9635.png,Normal/images/Normal-9635.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9635.png,Normal/masks/Normal-9635.png,True,False,299,299,L,148.97,aaefc98a409c41fc217988a081aff253
Normal-9636.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9636.png,Normal/images/Normal-9636.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9636.png,Normal/masks/Normal-9636.png,True,False,299,299,L,110.78,645b7fd71a750e864978c52b10a4aa32
Normal-9637.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9637.png,Normal/images/Normal-9637.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9637.png,Normal/masks/Normal-9637.png,True,False,299,299,L,134.2,9f7c442765d49f8530b28f676e0f86b7
Normal-9638.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9638.png,Normal/images/Normal-9638.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9638.png,Normal/masks/Normal-9638.png,True,False,299,299,L,151.48,1698b332a2b05ee0247c561d0f7cbd11
Normal-9639.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9639.png,Normal/images/Normal-9639.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9639.png,Normal/masks/Normal-9639.png,True,False,299,299,L,104.82,be67736e20e656877206fba305efc1f1
Normal-964.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-964.png,Normal/images/Normal-964.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-964.png,Normal/masks/Normal-964.png,True,False,299,299,L,112.96,be82857edf598018cfec6f86c14ca672
Normal-9640.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9640.png,Normal/images/Normal-9640.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9640.png,Normal/masks/Normal-9640.png,True,False,299,299,L,117.83,518656365ef31ee0db6d904ee63104eb
Normal-9641.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9641.png,Normal/images/Normal-9641.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9641.png,Normal/masks/Normal-9641.png,True,False,299,299,L,135.43,6f7bcc23b50aa8c7f3ce71f00a4c0476
Normal-9642.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9642.png,Normal/images/Normal-9642.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9642.png,Normal/masks/Normal-9642.png,True,False,299,299,L,171.74,2aa54be9818be6957cc902a446aed9b2
Normal-9643.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9643.png,Normal/images/Normal-9643.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9643.png,Normal/masks/Normal-9643.png,True,False,299,299,L,128.82,a54c3b1cd86dac379d989f3379ee70cb
Normal-9644.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9644.png,Normal/images/Normal-9644.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9644.png,Normal/masks/Normal-9644.png,True,False,299,299,L,141.4,02143963fb8b0cbaac21e2665a5021ba
Normal-9645.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9645.png,Normal/images/Normal-9645.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9645.png,Normal/masks/Normal-9645.png,True,False,299,299,L,103.55,f7fc81acce073d88bbcf44b6bc7a7e36
Normal-9646.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9646.png,Normal/images/Normal-9646.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9646.png,Normal/masks/Normal-9646.png,True,False,299,299,L,148.94,2dd1e35f7eed29573fe012a70e737b06
Normal-9647.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9647.png,Normal/images/Normal-9647.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9647.png,Normal/masks/Normal-9647.png,True,False,299,299,L,100.33,a728118f4a5843791d20973d440adbe6
Normal-9648.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9648.png,Normal/images/Normal-9648.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9648.png,Normal/masks/Normal-9648.png,True,False,299,299,L,140.93,9fe3e4f9109bff8ea5ea051e6fb7f80c
Normal-9649.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9649.png,Normal/images/Normal-9649.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9649.png,Normal/masks/Normal-9649.png,True,False,299,299,L,152.99,93457d7a7cf03c7876a4f9abd6030aea
Normal-965.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-965.png,Normal/images/Normal-965.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-965.png,Normal/masks/Normal-965.png,True,False,299,299,L,119.74,c53e974a707cd96f7f9a4a4ffe2383bb
Normal-9650.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9650.png,Normal/images/Normal-9650.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9650.png,Normal/masks/Normal-9650.png,True,False,299,299,L,131.75,574b7292a5bce99d3c181a883984ec11
Normal-9651.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9651.png,Normal/images/Normal-9651.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9651.png,Normal/masks/Normal-9651.png,True,False,299,299,L,161.39,dc5766f8942377642fd644ec0bdf8022
Normal-9652.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9652.png,Normal/images/Normal-9652.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9652.png,Normal/masks/Normal-9652.png,True,False,299,299,L,132.36,3dce156698dafed6876591cf694e5ccc
Normal-9653.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9653.png,Normal/images/Normal-9653.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9653.png,Normal/masks/Normal-9653.png,True,False,299,299,L,114.23,877ec57ef1a16dccf3af1b4fc683e9f7
Normal-9654.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9654.png,Normal/images/Normal-9654.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9654.png,Normal/masks/Normal-9654.png,True,False,299,299,L,118.23,75683f74d954d6bf9143bb30a75393b8
Normal-9655.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9655.png,Normal/images/Normal-9655.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9655.png,Normal/masks/Normal-9655.png,True,False,299,299,L,186.14,85ad64784b6ca29ac870a249d08b2833
Normal-9656.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9656.png,Normal/images/Normal-9656.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9656.png,Normal/masks/Normal-9656.png,True,False,299,299,L,115.5,4c850dad06ef0f056c30e97755e73ef1
Normal-9657.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9657.png,Normal/images/Normal-9657.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9657.png,Normal/masks/Normal-9657.png,True,False,299,299,L,122.87,05547af57d2a73dc05482c2a07984e8e
Normal-9658.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9658.png,Normal/images/Normal-9658.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9658.png,Normal/masks/Normal-9658.png,True,False,299,299,L,94.12,e96c7622c3034c3d23268ee0afa51e4b
Normal-9659.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9659.png,Normal/images/Normal-9659.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9659.png,Normal/masks/Normal-9659.png,True,False,299,299,L,117.79,b1e6a8f9cb0f795f7a5371f1a702c3b9
Normal-966.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-966.png,Normal/images/Normal-966.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-966.png,Normal/masks/Normal-966.png,True,False,299,299,L,109.91,6d3066aef35f5c1cfa78c4209075ada4
Normal-9660.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9660.png,Normal/images/Normal-9660.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9660.png,Normal/masks/Normal-9660.png,True,False,299,299,L,134.34,9db0663c4a49653cbbeec96c8366f34f
Normal-9661.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9661.png,Normal/images/Normal-9661.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9661.png,Normal/masks/Normal-9661.png,True,False,299,299,L,158.82,f10a067ec332e240c20a18dcf742ceda
Normal-9662.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9662.png,Normal/images/Normal-9662.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9662.png,Normal/masks/Normal-9662.png,True,False,299,299,L,109.12,dde8c01f41e747d34d550763b4cf23a6
Normal-9663.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9663.png,Normal/images/Normal-9663.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9663.png,Normal/masks/Normal-9663.png,True,False,299,299,L,107.94,8f14e8b853d6501c66f4a656128b3499
Normal-9664.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9664.png,Normal/images/Normal-9664.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9664.png,Normal/masks/Normal-9664.png,True,False,299,299,L,112.04,13283200f9ceadf461f9dc64ce916b84
Normal-9665.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9665.png,Normal/images/Normal-9665.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9665.png,Normal/masks/Normal-9665.png,True,False,299,299,L,160.76,c504c27180704be8f64375f13b645b23
Normal-9666.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9666.png,Normal/images/Normal-9666.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9666.png,Normal/masks/Normal-9666.png,True,False,299,299,L,143.24,cb7092795801e1c8b55b59ba8daa7bcd
Normal-9667.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9667.png,Normal/images/Normal-9667.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9667.png,Normal/masks/Normal-9667.png,True,False,299,299,L,80.91,0569841c01de220f20c0776becd54e67
Normal-9668.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9668.png,Normal/images/Normal-9668.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9668.png,Normal/masks/Normal-9668.png,True,False,299,299,L,150.28,d8524da3e6da55e7a6100699c331eb8e
Normal-9669.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9669.png,Normal/images/Normal-9669.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9669.png,Normal/masks/Normal-9669.png,True,False,299,299,L,118.6,c3e686db2c2e0e49850b0320bb847a09
Normal-967.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-967.png,Normal/images/Normal-967.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-967.png,Normal/masks/Normal-967.png,True,False,299,299,L,141.35,6f6b39603eb1f9e502960ac57df0f750
Normal-9670.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9670.png,Normal/images/Normal-9670.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9670.png,Normal/masks/Normal-9670.png,True,False,299,299,L,143.99,c7e974e7556596356dd7c645df04227a
Normal-9671.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9671.png,Normal/images/Normal-9671.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9671.png,Normal/masks/Normal-9671.png,True,False,299,299,L,149.84,1ef28394d71b615d1db628d7569c9ffc
Normal-9672.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9672.png,Normal/images/Normal-9672.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9672.png,Normal/masks/Normal-9672.png,True,False,299,299,L,116.01,5bb5a5445861286b598220813d45242e
Normal-9673.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9673.png,Normal/images/Normal-9673.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9673.png,Normal/masks/Normal-9673.png,True,False,299,299,L,147.33,5d7156d175d6ffe1fe392b201db5ed9f
Normal-9674.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9674.png,Normal/images/Normal-9674.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9674.png,Normal/masks/Normal-9674.png,True,False,299,299,L,143.22,83cfa133eb4da302604246f731b44235
Normal-9675.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9675.png,Normal/images/Normal-9675.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9675.png,Normal/masks/Normal-9675.png,True,False,299,299,L,105.68,b1feb88b980ba40aae3846a5321d1b5a
Normal-9676.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9676.png,Normal/images/Normal-9676.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9676.png,Normal/masks/Normal-9676.png,True,False,299,299,L,140.0,779587f2d7d6273c2785fbdf56c7c089
Normal-9677.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9677.png,Normal/images/Normal-9677.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9677.png,Normal/masks/Normal-9677.png,True,False,299,299,L,134.87,a76913c62098bf3f9190fe1cf2c8309e
Normal-9678.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9678.png,Normal/images/Normal-9678.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9678.png,Normal/masks/Normal-9678.png,True,False,299,299,L,118.61,60407e59055dce76bf8fae583be9da55
Normal-9679.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9679.png,Normal/images/Normal-9679.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9679.png,Normal/masks/Normal-9679.png,True,False,299,299,L,157.19,bb6431d9325c78c004627d9f3d3b8a56
Normal-968.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-968.png,Normal/images/Normal-968.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-968.png,Normal/masks/Normal-968.png,True,False,299,299,L,98.9,af73fff083e02e82eb19aaefb78bf95a
Normal-9680.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9680.png,Normal/images/Normal-9680.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9680.png,Normal/masks/Normal-9680.png,True,False,299,299,L,103.05,23792013b428dc75b13b6b245ebf69c7
Normal-9681.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9681.png,Normal/images/Normal-9681.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9681.png,Normal/masks/Normal-9681.png,True,False,299,299,L,133.21,4f7ac34c4a0b47592ca732b835a56394
Normal-9682.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9682.png,Normal/images/Normal-9682.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9682.png,Normal/masks/Normal-9682.png,True,False,299,299,L,122.23,d2ae44bfb6e2b917784ce63369a1bdfa
Normal-9683.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9683.png,Normal/images/Normal-9683.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9683.png,Normal/masks/Normal-9683.png,True,False,299,299,L,108.68,aef3b2669c915a3d2b662c6507221d6b
Normal-9684.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9684.png,Normal/images/Normal-9684.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9684.png,Normal/masks/Normal-9684.png,True,False,299,299,L,185.97,7508b61b63a0bca6101cf4ef35d95e1c
Normal-9685.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9685.png,Normal/images/Normal-9685.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9685.png,Normal/masks/Normal-9685.png,True,False,299,299,L,134.0,4c300c9d3396d82a8baa81d52971e003
Normal-9686.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9686.png,Normal/images/Normal-9686.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9686.png,Normal/masks/Normal-9686.png,True,False,299,299,L,129.55,6183042065263eb0e00ff150fc63d643
Normal-9687.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9687.png,Normal/images/Normal-9687.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9687.png,Normal/masks/Normal-9687.png,True,False,299,299,L,143.51,d362b39241709b0b27f581de469413c5
Normal-9688.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9688.png,Normal/images/Normal-9688.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9688.png,Normal/masks/Normal-9688.png,True,False,299,299,L,127.09,feaaa72998d3d223b5572f47d579b49a
Normal-9689.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9689.png,Normal/images/Normal-9689.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9689.png,Normal/masks/Normal-9689.png,True,False,299,299,L,147.33,46c05734110566490218fe7e41f1a376
Normal-969.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-969.png,Normal/images/Normal-969.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-969.png,Normal/masks/Normal-969.png,True,False,299,299,L,127.52,6b53acb16cba8294839e50e74874806d
Normal-9690.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9690.png,Normal/images/Normal-9690.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9690.png,Normal/masks/Normal-9690.png,True,False,299,299,L,124.49,2d961112eeb88f93faefb36e10e624ca
Normal-9691.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9691.png,Normal/images/Normal-9691.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9691.png,Normal/masks/Normal-9691.png,True,False,299,299,L,97.57,1fa3cc3b6592ea7dd09036006eb78174
Normal-9692.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9692.png,Normal/images/Normal-9692.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9692.png,Normal/masks/Normal-9692.png,True,False,299,299,L,128.03,1a730980fd7e1d1e7dfe2a5d82a82c46
Normal-9693.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9693.png,Normal/images/Normal-9693.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9693.png,Normal/masks/Normal-9693.png,True,False,299,299,L,160.02,06fb40fa6d5a700ce51be4c1b4ca5581
Normal-9694.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9694.png,Normal/images/Normal-9694.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9694.png,Normal/masks/Normal-9694.png,True,False,299,299,L,162.2,d39107b85f09999984c67810e2883eb3
Normal-9695.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9695.png,Normal/images/Normal-9695.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9695.png,Normal/masks/Normal-9695.png,True,False,299,299,L,144.16,c8baed19812ec36013eafe4d3794b868
Normal-9696.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9696.png,Normal/images/Normal-9696.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9696.png,Normal/masks/Normal-9696.png,True,False,299,299,L,130.2,4445970e753e904b06e46b42ddb95467
Normal-9697.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9697.png,Normal/images/Normal-9697.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9697.png,Normal/masks/Normal-9697.png,True,False,299,299,L,174.72,cfdc9844c9b597fff109041fd139fa6d
Normal-9698.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9698.png,Normal/images/Normal-9698.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9698.png,Normal/masks/Normal-9698.png,True,False,299,299,L,149.39,b87b7e44836de06e18b369d294375909
Normal-9699.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9699.png,Normal/images/Normal-9699.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9699.png,Normal/masks/Normal-9699.png,True,False,299,299,L,175.74,79073a521262f01da9aa4b6d1d322bb8
Normal-97.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-97.png,Normal/images/Normal-97.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-97.png,Normal/masks/Normal-97.png,True,False,299,299,L,135.86,87ef115adde4fa36c50ba13a4ec753ad
Normal-970.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-970.png,Normal/images/Normal-970.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-970.png,Normal/masks/Normal-970.png,True,False,299,299,L,122.78,096b9b15a75393401f0765df4ab32f52
Normal-9700.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9700.png,Normal/images/Normal-9700.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9700.png,Normal/masks/Normal-9700.png,True,False,299,299,L,118.8,0dd9637a406a3dc4da24bfa4360ea3a3
Normal-9701.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9701.png,Normal/images/Normal-9701.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9701.png,Normal/masks/Normal-9701.png,True,False,299,299,L,132.1,703e42f433352b39442fe02b3ba08b46
Normal-9702.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9702.png,Normal/images/Normal-9702.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9702.png,Normal/masks/Normal-9702.png,True,False,299,299,L,112.77,12f5833211df6a73a170abe0ffe20750
Normal-9703.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9703.png,Normal/images/Normal-9703.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9703.png,Normal/masks/Normal-9703.png,True,False,299,299,L,116.01,0005fd114eb9934999952383375819c0
Normal-9704.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9704.png,Normal/images/Normal-9704.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9704.png,Normal/masks/Normal-9704.png,True,False,299,299,L,125.2,6ada7be60da54f7225ba3869114dbb53
Normal-9705.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9705.png,Normal/images/Normal-9705.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9705.png,Normal/masks/Normal-9705.png,True,False,299,299,L,120.65,077fc0985168b82fc6a56067c231fccd
Normal-9706.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9706.png,Normal/images/Normal-9706.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9706.png,Normal/masks/Normal-9706.png,True,False,299,299,L,133.41,06cf2730fabc8fa2c4b470ae6f3a4028
Normal-9707.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9707.png,Normal/images/Normal-9707.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9707.png,Normal/masks/Normal-9707.png,True,False,299,299,L,114.83,8416afbf4335d3ce74b715fb32b2e35e
Normal-9708.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9708.png,Normal/images/Normal-9708.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9708.png,Normal/masks/Normal-9708.png,True,False,299,299,L,119.83,4454ca492f35c167358c33c8943b7359
Normal-9709.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9709.png,Normal/images/Normal-9709.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9709.png,Normal/masks/Normal-9709.png,True,False,299,299,L,142.8,398bce8752f09e86c8b58c41de7e2679
Normal-971.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-971.png,Normal/images/Normal-971.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-971.png,Normal/masks/Normal-971.png,True,False,299,299,L,125.66,e0e249bfadecb0d84d7a875304e03533
Normal-9710.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9710.png,Normal/images/Normal-9710.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9710.png,Normal/masks/Normal-9710.png,True,False,299,299,L,106.14,5410d3674a80f3ad744f07153b06c1dc
Normal-9711.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9711.png,Normal/images/Normal-9711.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9711.png,Normal/masks/Normal-9711.png,True,False,299,299,L,124.48,9b5e1c63704b58eceb35c2e4bb1d07c0
Normal-9712.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9712.png,Normal/images/Normal-9712.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9712.png,Normal/masks/Normal-9712.png,True,False,299,299,L,127.35,64b687e0d9e89f034a6f7c6384eeb025
Normal-9713.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9713.png,Normal/images/Normal-9713.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9713.png,Normal/masks/Normal-9713.png,True,False,299,299,L,117.43,742c20ab29031cda417e48ef091ef03f
Normal-9714.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9714.png,Normal/images/Normal-9714.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9714.png,Normal/masks/Normal-9714.png,True,False,299,299,L,93.01,097028875c547e7ebb94742ef2d67ec0
Normal-9715.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9715.png,Normal/images/Normal-9715.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9715.png,Normal/masks/Normal-9715.png,True,False,299,299,L,117.73,02c228d7659aa37cae05eca050cf544a
Normal-9716.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9716.png,Normal/images/Normal-9716.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9716.png,Normal/masks/Normal-9716.png,True,False,299,299,L,108.87,7c7d8c7c5996bc192b6b0a5ef5ab9f51
Normal-9717.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9717.png,Normal/images/Normal-9717.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9717.png,Normal/masks/Normal-9717.png,True,False,299,299,L,116.04,a4b3d11193242e670f5ac8d24af91f13
Normal-9718.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9718.png,Normal/images/Normal-9718.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9718.png,Normal/masks/Normal-9718.png,True,False,299,299,L,146.42,5f289ef39c7e92c7f8b3accf0796c55d
Normal-9719.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9719.png,Normal/images/Normal-9719.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9719.png,Normal/masks/Normal-9719.png,True,False,299,299,L,173.74,bed27962de078a30048d54f6c5fb62d0
Normal-972.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-972.png,Normal/images/Normal-972.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-972.png,Normal/masks/Normal-972.png,True,False,299,299,L,116.11,c04791cdbe5f7b4eb888c121fa05e375
Normal-9720.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9720.png,Normal/images/Normal-9720.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9720.png,Normal/masks/Normal-9720.png,True,False,299,299,L,111.02,440457e45f02a3abd3c3c9d0cf8028f5
Normal-9721.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9721.png,Normal/images/Normal-9721.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9721.png,Normal/masks/Normal-9721.png,True,False,299,299,L,118.25,d5017a8d8b618f2f5a3a131a07072a5c
Normal-9722.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9722.png,Normal/images/Normal-9722.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9722.png,Normal/masks/Normal-9722.png,True,False,299,299,L,132.65,f17d66313965c21a5a6841114c0370e7
Normal-9723.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9723.png,Normal/images/Normal-9723.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9723.png,Normal/masks/Normal-9723.png,True,False,299,299,L,118.72,30b4540e2f03b879fc4b8b862664a7e0
Normal-9724.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9724.png,Normal/images/Normal-9724.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9724.png,Normal/masks/Normal-9724.png,True,False,299,299,L,157.49,ea0f5e989e28b254721ac30ba6b3e50e
Normal-9725.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9725.png,Normal/images/Normal-9725.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9725.png,Normal/masks/Normal-9725.png,True,False,299,299,L,167.62,5dcc3c2d5f6ff8baccffc65e3018fe39
Normal-9726.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9726.png,Normal/images/Normal-9726.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9726.png,Normal/masks/Normal-9726.png,True,False,299,299,L,124.16,3515c66f5a81f30c032813aff735442d
Normal-9727.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9727.png,Normal/images/Normal-9727.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9727.png,Normal/masks/Normal-9727.png,True,False,299,299,L,129.9,3b1c4289ca5f621f5983c453fad6a660
Normal-9728.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9728.png,Normal/images/Normal-9728.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9728.png,Normal/masks/Normal-9728.png,True,False,299,299,L,112.54,1fb5ad6ddebb6d3666085d19aa0ecc37
Normal-9729.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9729.png,Normal/images/Normal-9729.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9729.png,Normal/masks/Normal-9729.png,True,False,299,299,L,113.24,31e026a1e3391d29f91df45a983cd371
Normal-973.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-973.png,Normal/images/Normal-973.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-973.png,Normal/masks/Normal-973.png,True,False,299,299,L,126.23,5dfb0d56e2eb75016496c29c6fda7a6d
Normal-9730.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9730.png,Normal/images/Normal-9730.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9730.png,Normal/masks/Normal-9730.png,True,False,299,299,L,126.49,5e125e249bd8019e437375034956d425
Normal-9731.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9731.png,Normal/images/Normal-9731.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9731.png,Normal/masks/Normal-9731.png,True,False,299,299,L,150.96,321a641cbb6806313ace22d19c862358
Normal-9732.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9732.png,Normal/images/Normal-9732.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9732.png,Normal/masks/Normal-9732.png,True,False,299,299,L,106.33,1fb717b0a1f75b1756085a37eae32ffa
Normal-9733.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9733.png,Normal/images/Normal-9733.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9733.png,Normal/masks/Normal-9733.png,True,False,299,299,L,128.93,846fbb3c5fb2d887ae00796c5ea01505
Normal-9734.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9734.png,Normal/images/Normal-9734.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9734.png,Normal/masks/Normal-9734.png,True,False,299,299,L,115.92,d6b8e638199b8b3fc4912c4823593952
Normal-9735.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9735.png,Normal/images/Normal-9735.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9735.png,Normal/masks/Normal-9735.png,True,False,299,299,L,104.63,6493165785faaced35ec1a66b0110488
Normal-9736.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9736.png,Normal/images/Normal-9736.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9736.png,Normal/masks/Normal-9736.png,True,False,299,299,L,138.36,e2b2b95f7214d81180808e1daf6aa0b0
Normal-9737.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9737.png,Normal/images/Normal-9737.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9737.png,Normal/masks/Normal-9737.png,True,False,299,299,L,117.11,d4efafbd9ee044fd9568ca539f32b1b2
Normal-9738.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9738.png,Normal/images/Normal-9738.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9738.png,Normal/masks/Normal-9738.png,True,False,299,299,L,165.59,9adc6bd94d29198f236ee8d96893cabe
Normal-9739.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9739.png,Normal/images/Normal-9739.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9739.png,Normal/masks/Normal-9739.png,True,False,299,299,L,132.59,a1953dad7834e1b5be192fe4af3a7a9a
Normal-974.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-974.png,Normal/images/Normal-974.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-974.png,Normal/masks/Normal-974.png,True,False,299,299,L,121.16,c55a1e0025c3aee1c26b70ae4d48d1f3
Normal-9740.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9740.png,Normal/images/Normal-9740.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9740.png,Normal/masks/Normal-9740.png,True,False,299,299,L,119.28,1d01540aa76ca7e190f2a50e0d02cac6
Normal-9741.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9741.png,Normal/images/Normal-9741.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9741.png,Normal/masks/Normal-9741.png,True,False,299,299,L,110.66,d0d4d871035a5723a2ba61e02bb29ff2
Normal-9742.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9742.png,Normal/images/Normal-9742.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9742.png,Normal/masks/Normal-9742.png,True,False,299,299,L,151.63,51fb38ddcc669baf608c7212f1afddec
Normal-9743.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9743.png,Normal/images/Normal-9743.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9743.png,Normal/masks/Normal-9743.png,True,False,299,299,L,152.6,67f1ebea815972c09a516e848b8d71b5
Normal-9744.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9744.png,Normal/images/Normal-9744.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9744.png,Normal/masks/Normal-9744.png,True,False,299,299,L,123.1,05285c6e1e2323c64261c5fdc8a3c84b
Normal-9745.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9745.png,Normal/images/Normal-9745.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9745.png,Normal/masks/Normal-9745.png,True,False,299,299,L,125.35,c19668a9272bcee4a90e070ef140f9c6
Normal-9746.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9746.png,Normal/images/Normal-9746.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9746.png,Normal/masks/Normal-9746.png,True,False,299,299,L,146.07,d0d5241ac34751bef5f85ba8bcc8a3a8
Normal-9747.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9747.png,Normal/images/Normal-9747.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9747.png,Normal/masks/Normal-9747.png,True,False,299,299,L,106.68,4a009126bd532c126e172133f9b8db8c
Normal-9748.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9748.png,Normal/images/Normal-9748.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9748.png,Normal/masks/Normal-9748.png,True,False,299,299,L,142.01,176db74aad73dd9f3d40d77f4f017706
Normal-9749.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9749.png,Normal/images/Normal-9749.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9749.png,Normal/masks/Normal-9749.png,True,False,299,299,L,102.56,35568c060646c9968f2c578d7634470a
Normal-975.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-975.png,Normal/images/Normal-975.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-975.png,Normal/masks/Normal-975.png,True,False,299,299,L,134.56,c70a71f356fa24044e043551bf4c657c
Normal-9750.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9750.png,Normal/images/Normal-9750.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9750.png,Normal/masks/Normal-9750.png,True,False,299,299,L,89.46,bfd0884e4e2bab0025c392c8308ee485
Normal-9751.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9751.png,Normal/images/Normal-9751.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9751.png,Normal/masks/Normal-9751.png,True,False,299,299,L,89.74,50652b00ce57a264a730e20bfded718b
Normal-9752.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9752.png,Normal/images/Normal-9752.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9752.png,Normal/masks/Normal-9752.png,True,False,299,299,L,107.83,fa4a7a610f84a5fb26d30d722be80f6e
Normal-9753.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9753.png,Normal/images/Normal-9753.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9753.png,Normal/masks/Normal-9753.png,True,False,299,299,L,113.45,b2412d31056e36c100480eedbf5ac7b9
Normal-9754.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9754.png,Normal/images/Normal-9754.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9754.png,Normal/masks/Normal-9754.png,True,False,299,299,L,145.78,7ec4f8119fe75f3276b5a811197058ba
Normal-9755.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9755.png,Normal/images/Normal-9755.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9755.png,Normal/masks/Normal-9755.png,True,False,299,299,L,118.46,0b4bfee16c9062479048fce400d419d5
Normal-9756.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9756.png,Normal/images/Normal-9756.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9756.png,Normal/masks/Normal-9756.png,True,False,299,299,L,160.41,29aa4014e7b15d7741a06d1bd7d164ea
Normal-9757.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9757.png,Normal/images/Normal-9757.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9757.png,Normal/masks/Normal-9757.png,True,False,299,299,L,122.38,b5ac5ac2c578015d1a15890a56303039
Normal-9758.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9758.png,Normal/images/Normal-9758.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9758.png,Normal/masks/Normal-9758.png,True,False,299,299,L,111.06,4e39c273e1f53e5985b7b0f5ab3d115f
Normal-9759.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9759.png,Normal/images/Normal-9759.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9759.png,Normal/masks/Normal-9759.png,True,False,299,299,L,109.4,4da6d897329e2e6f13c9040c01b5d670
Normal-976.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-976.png,Normal/images/Normal-976.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-976.png,Normal/masks/Normal-976.png,True,False,299,299,L,122.72,62aa119bd26b704ce3b3c5c55f614f80
Normal-9760.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9760.png,Normal/images/Normal-9760.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9760.png,Normal/masks/Normal-9760.png,True,False,299,299,L,167.25,d73c574924e235cb1bd339cacd518421
Normal-9761.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9761.png,Normal/images/Normal-9761.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9761.png,Normal/masks/Normal-9761.png,True,False,299,299,L,137.65,bf6fb93ad3711916af7a5837cf03e751
Normal-9762.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9762.png,Normal/images/Normal-9762.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9762.png,Normal/masks/Normal-9762.png,True,False,299,299,L,111.97,8d13a2a98744481554093a2cb84cc962
Normal-9763.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9763.png,Normal/images/Normal-9763.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9763.png,Normal/masks/Normal-9763.png,True,False,299,299,L,116.42,4761363da04dd5e417ccfcc30ac8d8e2
Normal-9764.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9764.png,Normal/images/Normal-9764.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9764.png,Normal/masks/Normal-9764.png,True,False,299,299,L,131.89,dddd469f96780bcc22e51c7c79f569c1
Normal-9765.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9765.png,Normal/images/Normal-9765.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9765.png,Normal/masks/Normal-9765.png,True,False,299,299,L,114.73,f91af9f7e5c205fc605a0d62b956cd1e
Normal-9766.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9766.png,Normal/images/Normal-9766.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9766.png,Normal/masks/Normal-9766.png,True,False,299,299,L,118.02,ec7e7729142dfc55c43ff3ff0f104548
Normal-9767.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9767.png,Normal/images/Normal-9767.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9767.png,Normal/masks/Normal-9767.png,True,False,299,299,L,118.67,70f1b86dd0b3e9d284a2fce08ccfd217
Normal-9768.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9768.png,Normal/images/Normal-9768.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9768.png,Normal/masks/Normal-9768.png,True,False,299,299,L,118.86,01aa6a7745afba9da9609dc567e30f84
Normal-9769.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9769.png,Normal/images/Normal-9769.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9769.png,Normal/masks/Normal-9769.png,True,False,299,299,L,102.5,edbde228e033e91a47d141f11412a4a6
Normal-977.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-977.png,Normal/images/Normal-977.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-977.png,Normal/masks/Normal-977.png,True,False,299,299,L,126.86,df7f74cc65c6afcdd99d220e31c40dc2
Normal-9770.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9770.png,Normal/images/Normal-9770.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9770.png,Normal/masks/Normal-9770.png,True,False,299,299,L,113.41,fb4356b5aef6141098d29b83698c0e9b
Normal-9771.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9771.png,Normal/images/Normal-9771.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9771.png,Normal/masks/Normal-9771.png,True,False,299,299,L,127.67,ab4200d769576ecc72a0c85ccbd0c777
Normal-9772.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9772.png,Normal/images/Normal-9772.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9772.png,Normal/masks/Normal-9772.png,True,False,299,299,L,118.22,35be0cb9ed6dc1d2d7924e8c2db8d55f
Normal-9773.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9773.png,Normal/images/Normal-9773.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9773.png,Normal/masks/Normal-9773.png,True,False,299,299,L,106.11,867cac8ed0104295a7bbd68a4888822c
Normal-9774.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9774.png,Normal/images/Normal-9774.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9774.png,Normal/masks/Normal-9774.png,True,False,299,299,L,135.5,e2c6370405c76cb878c00ce9a428010e
Normal-9775.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9775.png,Normal/images/Normal-9775.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9775.png,Normal/masks/Normal-9775.png,True,False,299,299,L,121.43,6a0f28717dd1a640ac122852f2a663e6
Normal-9776.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9776.png,Normal/images/Normal-9776.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9776.png,Normal/masks/Normal-9776.png,True,False,299,299,L,163.28,c915d713746dbe925fd2acb5ca998a1b
Normal-9777.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9777.png,Normal/images/Normal-9777.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9777.png,Normal/masks/Normal-9777.png,True,False,299,299,L,131.72,fc5f18391e773d7e56476474296f7753
Normal-9778.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9778.png,Normal/images/Normal-9778.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9778.png,Normal/masks/Normal-9778.png,True,False,299,299,L,126.76,94ca46a2ec5a23656baa0d5e7cad36cf
Normal-9779.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9779.png,Normal/images/Normal-9779.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9779.png,Normal/masks/Normal-9779.png,True,False,299,299,L,108.49,7c74c79b221fbd8d5e2cab562a988b31
Normal-978.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-978.png,Normal/images/Normal-978.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-978.png,Normal/masks/Normal-978.png,True,False,299,299,L,141.74,1a7ad9a005ea2f7713910a4e7c1d5d01
Normal-9780.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9780.png,Normal/images/Normal-9780.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9780.png,Normal/masks/Normal-9780.png,True,False,299,299,L,175.02,fb2181c1520347e7f83af338b2946731
Normal-9781.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9781.png,Normal/images/Normal-9781.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9781.png,Normal/masks/Normal-9781.png,True,False,299,299,L,121.12,9dc501a9f3c83cc94eaef70c3ec02f26
Normal-9782.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9782.png,Normal/images/Normal-9782.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9782.png,Normal/masks/Normal-9782.png,True,False,299,299,L,159.21,30b43f636c710473d94cc7055f07587c
Normal-9783.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9783.png,Normal/images/Normal-9783.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9783.png,Normal/masks/Normal-9783.png,True,False,299,299,L,130.93,f1582402774a67c3f8e98247ae8c0396
Normal-9784.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9784.png,Normal/images/Normal-9784.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9784.png,Normal/masks/Normal-9784.png,True,False,299,299,L,110.8,f9d81204aede981dc68803a095da1aeb
Normal-9785.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9785.png,Normal/images/Normal-9785.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9785.png,Normal/masks/Normal-9785.png,True,False,299,299,L,107.98,dcd2a2586508b9ca856ccb7d6ce8180b
Normal-9786.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9786.png,Normal/images/Normal-9786.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9786.png,Normal/masks/Normal-9786.png,True,False,299,299,L,109.58,e0ceae4f2bb012279a3dc8bd8dacd72a
Normal-9787.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9787.png,Normal/images/Normal-9787.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9787.png,Normal/masks/Normal-9787.png,True,False,299,299,L,130.84,337514366e320ef1a2ff4431d137f232
Normal-9788.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9788.png,Normal/images/Normal-9788.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9788.png,Normal/masks/Normal-9788.png,True,False,299,299,L,124.78,3e2ac40225c6e6b9b8af1f345d1fd153
Normal-9789.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9789.png,Normal/images/Normal-9789.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9789.png,Normal/masks/Normal-9789.png,True,False,299,299,L,125.06,63aef6595b65f61b89e638f26782cf02
Normal-979.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-979.png,Normal/images/Normal-979.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-979.png,Normal/masks/Normal-979.png,True,False,299,299,L,153.08,b002ba454638a4e6580650c70de89982
Normal-9790.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9790.png,Normal/images/Normal-9790.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9790.png,Normal/masks/Normal-9790.png,True,False,299,299,L,120.33,24c7bc51d5e7014662e8e475fc552614
Normal-9791.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9791.png,Normal/images/Normal-9791.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9791.png,Normal/masks/Normal-9791.png,True,False,299,299,L,129.35,e08b70c80b748b150e9c83d52fcaaee0
Normal-9792.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9792.png,Normal/images/Normal-9792.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9792.png,Normal/masks/Normal-9792.png,True,False,299,299,L,175.17,45b49c67fb616ecb859be1890b42b16b
Normal-9793.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9793.png,Normal/images/Normal-9793.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9793.png,Normal/masks/Normal-9793.png,True,False,299,299,L,100.6,9f8aef0a2bf1f2c221a0e1584752018f
Normal-9794.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9794.png,Normal/images/Normal-9794.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9794.png,Normal/masks/Normal-9794.png,True,False,299,299,L,131.77,823edecd7a71ba2625ff3c4b6e12c0c4
Normal-9795.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9795.png,Normal/images/Normal-9795.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9795.png,Normal/masks/Normal-9795.png,True,False,299,299,L,113.69,c4860a544fbd72bcb3e3154f4377cda2
Normal-9796.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9796.png,Normal/images/Normal-9796.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9796.png,Normal/masks/Normal-9796.png,True,False,299,299,L,134.04,8a78e47022ca34db30a1763e681c9710
Normal-9797.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9797.png,Normal/images/Normal-9797.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9797.png,Normal/masks/Normal-9797.png,True,False,299,299,L,129.19,b098439f969705d81e2327c91ac57466
Normal-9798.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9798.png,Normal/images/Normal-9798.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9798.png,Normal/masks/Normal-9798.png,True,False,299,299,L,128.8,8a71a160046c656279b53b1ff7dff77b
Normal-9799.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9799.png,Normal/images/Normal-9799.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9799.png,Normal/masks/Normal-9799.png,True,False,299,299,L,125.31,a181132ce1e3b2a1da16f9ecde9e0209
Normal-98.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-98.png,Normal/images/Normal-98.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-98.png,Normal/masks/Normal-98.png,True,False,299,299,L,119.32,1f7d9e3eff5d146839426d6b9ecfa26f
Normal-980.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-980.png,Normal/images/Normal-980.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-980.png,Normal/masks/Normal-980.png,True,False,299,299,L,162.27,a2c2b24b3b9fafb00e9f5a5b9658a4d0
Normal-9800.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9800.png,Normal/images/Normal-9800.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9800.png,Normal/masks/Normal-9800.png,True,False,299,299,L,125.88,da80a9bc435e09f23bf199fadd8ecfe5
Normal-9801.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9801.png,Normal/images/Normal-9801.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9801.png,Normal/masks/Normal-9801.png,True,False,299,299,L,141.41,76d4284486c67dc4b91b4489bd606570
Normal-9802.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9802.png,Normal/images/Normal-9802.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9802.png,Normal/masks/Normal-9802.png,True,False,299,299,L,155.7,ae0a9d2523fc02a5ef41a1d7ae7aff6b
Normal-9803.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9803.png,Normal/images/Normal-9803.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9803.png,Normal/masks/Normal-9803.png,True,False,299,299,L,122.91,b6ae96c61ed7104f70dceee106df16f8
Normal-9804.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9804.png,Normal/images/Normal-9804.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9804.png,Normal/masks/Normal-9804.png,True,False,299,299,L,122.33,0b8d5ecefe16947a0d1d84822da517cd
Normal-9805.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9805.png,Normal/images/Normal-9805.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9805.png,Normal/masks/Normal-9805.png,True,False,299,299,L,136.85,ad9a1096dee894e6a6152b87046b5b37
Normal-9806.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9806.png,Normal/images/Normal-9806.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9806.png,Normal/masks/Normal-9806.png,True,False,299,299,L,112.7,7c6d71f8138202edec9ab9c3f32466f0
Normal-9807.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9807.png,Normal/images/Normal-9807.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9807.png,Normal/masks/Normal-9807.png,True,False,299,299,L,174.36,e107c5d61040dfdca6d5639d6fc02230
Normal-9808.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9808.png,Normal/images/Normal-9808.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9808.png,Normal/masks/Normal-9808.png,True,False,299,299,L,165.66,a07fe4caaa256441ee1b6de2bf1be326
Normal-9809.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9809.png,Normal/images/Normal-9809.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9809.png,Normal/masks/Normal-9809.png,True,False,299,299,L,146.18,c60b69efac342d6c6adddff2ddd82b26
Normal-981.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-981.png,Normal/images/Normal-981.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-981.png,Normal/masks/Normal-981.png,True,False,299,299,L,104.56,f76c27b2ee8cee9be2b642c8caacab52
Normal-9810.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9810.png,Normal/images/Normal-9810.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9810.png,Normal/masks/Normal-9810.png,True,False,299,299,L,116.13,9311c987eab46605a3b014822edec637
Normal-9811.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9811.png,Normal/images/Normal-9811.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9811.png,Normal/masks/Normal-9811.png,True,False,299,299,L,127.77,5e5b931f428f144cb66d02e3f405e81d
Normal-9812.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9812.png,Normal/images/Normal-9812.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9812.png,Normal/masks/Normal-9812.png,True,False,299,299,L,115.21,c30e2a06aa3f0d2434a1a89278d96269
Normal-9813.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9813.png,Normal/images/Normal-9813.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9813.png,Normal/masks/Normal-9813.png,True,False,299,299,L,118.01,555c106a10e20618b2922e938bf11a5d
Normal-9814.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9814.png,Normal/images/Normal-9814.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9814.png,Normal/masks/Normal-9814.png,True,False,299,299,L,112.95,7a782f10605a78e17bb2e4848a33b5fb
Normal-9815.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9815.png,Normal/images/Normal-9815.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9815.png,Normal/masks/Normal-9815.png,True,False,299,299,L,167.77,192efcc4e2b05318f5c774b7c8af3c5f
Normal-9816.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9816.png,Normal/images/Normal-9816.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9816.png,Normal/masks/Normal-9816.png,True,False,299,299,L,138.76,5395bf99934e007cad7f219612bcc50d
Normal-9817.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9817.png,Normal/images/Normal-9817.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9817.png,Normal/masks/Normal-9817.png,True,False,299,299,L,132.49,c506c9897c6a01c11d33357fdb6681fb
Normal-9818.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9818.png,Normal/images/Normal-9818.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9818.png,Normal/masks/Normal-9818.png,True,False,299,299,L,97.19,628ffa84fb24f1b13ba4003b9db59222
Normal-9819.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9819.png,Normal/images/Normal-9819.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9819.png,Normal/masks/Normal-9819.png,True,False,299,299,L,111.08,7ee8afcd610a325235b7b2b10b413f96
Normal-982.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-982.png,Normal/images/Normal-982.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-982.png,Normal/masks/Normal-982.png,True,False,299,299,L,100.89,b86be9454a819a214904024cb3439538
Normal-9820.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9820.png,Normal/images/Normal-9820.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9820.png,Normal/masks/Normal-9820.png,True,False,299,299,L,123.42,e0f00923405cf8de1bdb8e0aa38c3bb5
Normal-9821.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9821.png,Normal/images/Normal-9821.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9821.png,Normal/masks/Normal-9821.png,True,False,299,299,L,94.38,4a940f0afbb09d58c61aecadd3d42306
Normal-9822.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9822.png,Normal/images/Normal-9822.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9822.png,Normal/masks/Normal-9822.png,True,False,299,299,L,140.52,e8d06f22c84185ed1fb2836d7bbd8511
Normal-9823.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9823.png,Normal/images/Normal-9823.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9823.png,Normal/masks/Normal-9823.png,True,False,299,299,L,170.01,1a45eecb715991cc9987875b496683ae
Normal-9824.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9824.png,Normal/images/Normal-9824.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9824.png,Normal/masks/Normal-9824.png,True,False,299,299,L,101.68,cf458e9ad26e08bc1e629db3d846918d
Normal-9825.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9825.png,Normal/images/Normal-9825.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9825.png,Normal/masks/Normal-9825.png,True,False,299,299,L,122.71,90070a96a774172ad1e11267fd061990
Normal-9826.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9826.png,Normal/images/Normal-9826.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9826.png,Normal/masks/Normal-9826.png,True,False,299,299,L,122.94,f319b73bc845e020634c87ef79172fe1
Normal-9827.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9827.png,Normal/images/Normal-9827.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9827.png,Normal/masks/Normal-9827.png,True,False,299,299,L,142.81,6e444ade57ad2e2d1e8a68447ccfcfbf
Normal-9828.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9828.png,Normal/images/Normal-9828.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9828.png,Normal/masks/Normal-9828.png,True,False,299,299,L,154.5,f32e02031ad3058b3dba6cc3fa8dccc9
Normal-9829.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9829.png,Normal/images/Normal-9829.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9829.png,Normal/masks/Normal-9829.png,True,False,299,299,L,112.79,38bdc4d7aa0fe8bb3ba8ca5918fef11d
Normal-983.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-983.png,Normal/images/Normal-983.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-983.png,Normal/masks/Normal-983.png,True,False,299,299,L,126.06,a9afe698128c7848ec019aa6f384c944
Normal-9830.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9830.png,Normal/images/Normal-9830.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9830.png,Normal/masks/Normal-9830.png,True,False,299,299,L,163.99,444093aa56d4f24852539415dd1230b9
Normal-9831.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9831.png,Normal/images/Normal-9831.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9831.png,Normal/masks/Normal-9831.png,True,False,299,299,L,158.62,d82de578a8262025222ab3fd4b413121
Normal-9832.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9832.png,Normal/images/Normal-9832.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9832.png,Normal/masks/Normal-9832.png,True,False,299,299,L,96.65,bc541d701432bc3e54194d4c8ff9ed60
Normal-9833.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9833.png,Normal/images/Normal-9833.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9833.png,Normal/masks/Normal-9833.png,True,False,299,299,L,120.5,f6a906781da8ab3bd346f6b1987279c5
Normal-9834.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9834.png,Normal/images/Normal-9834.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9834.png,Normal/masks/Normal-9834.png,True,False,299,299,L,119.79,ec4229304c8f8411242d958a580ee4de
Normal-9835.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9835.png,Normal/images/Normal-9835.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9835.png,Normal/masks/Normal-9835.png,True,False,299,299,L,113.8,ca03f8f73180f6ea42a9da205437793c
Normal-9836.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9836.png,Normal/images/Normal-9836.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9836.png,Normal/masks/Normal-9836.png,True,False,299,299,L,151.02,5403556ab8187a6f2abd7089a4afe0fc
Normal-9837.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9837.png,Normal/images/Normal-9837.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9837.png,Normal/masks/Normal-9837.png,True,False,299,299,L,133.45,6032b46674105348fafaf218f127a210
Normal-9838.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9838.png,Normal/images/Normal-9838.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9838.png,Normal/masks/Normal-9838.png,True,False,299,299,L,120.14,9923ae177548a7db635af0707a9f61eb
Normal-9839.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9839.png,Normal/images/Normal-9839.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9839.png,Normal/masks/Normal-9839.png,True,False,299,299,L,102.91,9e4110e695a6b580ff4c07d21cf79dbb
Normal-984.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-984.png,Normal/images/Normal-984.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-984.png,Normal/masks/Normal-984.png,True,False,299,299,L,104.5,59630df03275a28961925e66ff65b345
Normal-9840.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9840.png,Normal/images/Normal-9840.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9840.png,Normal/masks/Normal-9840.png,True,False,299,299,L,102.26,86dcced9f2c2280b81b8c33658031d5b
Normal-9841.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9841.png,Normal/images/Normal-9841.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9841.png,Normal/masks/Normal-9841.png,True,False,299,299,L,86.1,1123cd16c032858a446960867ca1a763
Normal-9842.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9842.png,Normal/images/Normal-9842.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9842.png,Normal/masks/Normal-9842.png,True,False,299,299,L,112.51,1460a8bb78948734b5d0361829fcb512
Normal-9843.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9843.png,Normal/images/Normal-9843.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9843.png,Normal/masks/Normal-9843.png,True,False,299,299,L,109.86,96dc96170b0bdf0f6b4d2c02772c5e9b
Normal-9844.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9844.png,Normal/images/Normal-9844.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9844.png,Normal/masks/Normal-9844.png,True,False,299,299,L,121.86,0bde1f27619ef560a275d8dbc54601d4
Normal-9845.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9845.png,Normal/images/Normal-9845.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9845.png,Normal/masks/Normal-9845.png,True,False,299,299,L,120.56,1deb250d91c5dfad9931621ec6616c39
Normal-9846.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9846.png,Normal/images/Normal-9846.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9846.png,Normal/masks/Normal-9846.png,True,False,299,299,L,147.83,d35bced31e8c02b1dba8fd95caf8e49d
Normal-9847.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9847.png,Normal/images/Normal-9847.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9847.png,Normal/masks/Normal-9847.png,True,False,299,299,L,111.58,86ba50176dbb77ba82ec1fce4356cad7
Normal-9848.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9848.png,Normal/images/Normal-9848.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9848.png,Normal/masks/Normal-9848.png,True,False,299,299,L,116.14,18ec1ac777c6c406966a8f29f238114d
Normal-9849.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9849.png,Normal/images/Normal-9849.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9849.png,Normal/masks/Normal-9849.png,True,False,299,299,L,106.88,da7c0969344930a1d5d9a66cdf91918c
Normal-985.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-985.png,Normal/images/Normal-985.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-985.png,Normal/masks/Normal-985.png,True,False,299,299,L,124.64,68809b3d2c1ab40d5d18d25d7ddc52dd
Normal-9850.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9850.png,Normal/images/Normal-9850.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9850.png,Normal/masks/Normal-9850.png,True,False,299,299,L,132.1,3e45a220ed22e3b056c1541616febc97
Normal-9851.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9851.png,Normal/images/Normal-9851.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9851.png,Normal/masks/Normal-9851.png,True,False,299,299,L,150.8,e3ee3af0fcda02c66095b96fe16ded86
Normal-9852.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9852.png,Normal/images/Normal-9852.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9852.png,Normal/masks/Normal-9852.png,True,False,299,299,L,121.62,41ed3d4d09737449a16ade155da6855b
Normal-9853.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9853.png,Normal/images/Normal-9853.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9853.png,Normal/masks/Normal-9853.png,True,False,299,299,L,112.13,931206a0253547ce3e904e0555024769
Normal-9854.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9854.png,Normal/images/Normal-9854.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9854.png,Normal/masks/Normal-9854.png,True,False,299,299,L,159.99,7d9e2afc74d8bbbe88367af3cb293de4
Normal-9855.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9855.png,Normal/images/Normal-9855.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9855.png,Normal/masks/Normal-9855.png,True,False,299,299,L,125.79,34c80bd265ec639c2721bc5c9449389f
Normal-9856.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9856.png,Normal/images/Normal-9856.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9856.png,Normal/masks/Normal-9856.png,True,False,299,299,L,162.63,962b31d10b43b8ca3369d731eab8a22b
Normal-9857.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9857.png,Normal/images/Normal-9857.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9857.png,Normal/masks/Normal-9857.png,True,False,299,299,L,112.72,99528a3430c8a41a9d74436df4627f37
Normal-9858.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9858.png,Normal/images/Normal-9858.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9858.png,Normal/masks/Normal-9858.png,True,False,299,299,L,135.86,b3f5030d3e991a20cef726d54e4da88a
Normal-9859.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9859.png,Normal/images/Normal-9859.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9859.png,Normal/masks/Normal-9859.png,True,False,299,299,L,110.99,47d5aa0a05dc0f7bd021bf6b05db580c
Normal-986.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-986.png,Normal/images/Normal-986.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-986.png,Normal/masks/Normal-986.png,True,False,299,299,L,120.6,1f3afc0f368440ae0de5ef7faea75bb3
Normal-9860.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9860.png,Normal/images/Normal-9860.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9860.png,Normal/masks/Normal-9860.png,True,False,299,299,L,105.38,d7a190d39f877ef85d5c543b4ffb524f
Normal-9861.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9861.png,Normal/images/Normal-9861.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9861.png,Normal/masks/Normal-9861.png,True,False,299,299,L,134.73,0b3c2b34a35b4e34db3cf703244229db
Normal-9862.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9862.png,Normal/images/Normal-9862.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9862.png,Normal/masks/Normal-9862.png,True,False,299,299,L,139.42,31e26491d5a776df41bdfbd1a823a7a9
Normal-9863.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9863.png,Normal/images/Normal-9863.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9863.png,Normal/masks/Normal-9863.png,True,False,299,299,L,166.33,5935d7707ec70e2add794a272429742b
Normal-9864.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9864.png,Normal/images/Normal-9864.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9864.png,Normal/masks/Normal-9864.png,True,False,299,299,L,153.94,cb9c7ce89862c4fb0c9f5fff903194ab
Normal-9865.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9865.png,Normal/images/Normal-9865.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9865.png,Normal/masks/Normal-9865.png,True,False,299,299,L,121.18,dce66f87b8fa15a87f907ce2d40b759b
Normal-9866.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9866.png,Normal/images/Normal-9866.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9866.png,Normal/masks/Normal-9866.png,True,False,299,299,L,126.77,7711fc2614e293a321146117e614e7ca
Normal-9867.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9867.png,Normal/images/Normal-9867.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9867.png,Normal/masks/Normal-9867.png,True,False,299,299,L,160.59,ac9fa4f9270a148543ce40dfe69c6878
Normal-9868.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9868.png,Normal/images/Normal-9868.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9868.png,Normal/masks/Normal-9868.png,True,False,299,299,L,140.02,090c3a16834fd1ae262736103a393bec
Normal-9869.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9869.png,Normal/images/Normal-9869.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9869.png,Normal/masks/Normal-9869.png,True,False,299,299,L,105.49,a453d660b208c1c0bb9a7f6f8b5d87b2
Normal-987.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-987.png,Normal/images/Normal-987.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-987.png,Normal/masks/Normal-987.png,True,False,299,299,L,147.08,c39853da977d13888acd4798e9af73e4
Normal-9870.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9870.png,Normal/images/Normal-9870.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9870.png,Normal/masks/Normal-9870.png,True,False,299,299,L,101.18,7679d18c08756975a168e02af9903b14
Normal-9871.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9871.png,Normal/images/Normal-9871.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9871.png,Normal/masks/Normal-9871.png,True,False,299,299,L,114.1,b1582561836d595f5e3a13be206e0ddb
Normal-9872.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9872.png,Normal/images/Normal-9872.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9872.png,Normal/masks/Normal-9872.png,True,False,299,299,L,178.15,ef5298f0c227a69815e4b71999d9b717
Normal-9873.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9873.png,Normal/images/Normal-9873.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9873.png,Normal/masks/Normal-9873.png,True,False,299,299,L,103.92,949207d5acbe10ef9b60950b5e456812
Normal-9874.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9874.png,Normal/images/Normal-9874.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9874.png,Normal/masks/Normal-9874.png,True,False,299,299,L,138.92,ac3f9d9f61d1cd134559176b0b1a7a3e
Normal-9875.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9875.png,Normal/images/Normal-9875.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9875.png,Normal/masks/Normal-9875.png,True,False,299,299,L,172.6,c6ed8d50032e95da012962a55c95a51c
Normal-9876.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9876.png,Normal/images/Normal-9876.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9876.png,Normal/masks/Normal-9876.png,True,False,299,299,L,121.82,d083f0bca16b6bac6442b196380c0fe0
Normal-9877.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9877.png,Normal/images/Normal-9877.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9877.png,Normal/masks/Normal-9877.png,True,False,299,299,L,97.97,3c175c30606c673e433a91a2a1d26fc1
Normal-9878.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9878.png,Normal/images/Normal-9878.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9878.png,Normal/masks/Normal-9878.png,True,False,299,299,L,128.99,bb37bb5816fe083af0e304d2a03934ca
Normal-9879.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9879.png,Normal/images/Normal-9879.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9879.png,Normal/masks/Normal-9879.png,True,False,299,299,L,129.3,37808388c4dbe8848f1295c0c0ea15df
Normal-988.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-988.png,Normal/images/Normal-988.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-988.png,Normal/masks/Normal-988.png,True,False,299,299,L,109.32,79e6b0e29bfee6487dcd7add4c7d5fff
Normal-9880.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9880.png,Normal/images/Normal-9880.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9880.png,Normal/masks/Normal-9880.png,True,False,299,299,L,120.59,9c7b6e23d263c4d36d7f88a14f39635a
Normal-9881.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9881.png,Normal/images/Normal-9881.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9881.png,Normal/masks/Normal-9881.png,True,False,299,299,L,138.68,7f7c281af671474e84677f4e04b522b6
Normal-9882.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9882.png,Normal/images/Normal-9882.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9882.png,Normal/masks/Normal-9882.png,True,False,299,299,L,178.18,e53be0d340750530cd33e5d1d63a46cf
Normal-9883.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9883.png,Normal/images/Normal-9883.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9883.png,Normal/masks/Normal-9883.png,True,False,299,299,L,157.17,8ec442e0d9fc66201e4130604ede86f7
Normal-9884.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9884.png,Normal/images/Normal-9884.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9884.png,Normal/masks/Normal-9884.png,True,False,299,299,L,161.99,eb80467aab8231ca6947f1d8d2656f2a
Normal-9885.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9885.png,Normal/images/Normal-9885.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9885.png,Normal/masks/Normal-9885.png,True,False,299,299,L,108.8,1bb45a31ea9b8d34daa5680efc0f82f3
Normal-9886.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9886.png,Normal/images/Normal-9886.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9886.png,Normal/masks/Normal-9886.png,True,False,299,299,L,143.33,bc0d65cb212328aa49ee40ebd10c3f07
Normal-9887.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9887.png,Normal/images/Normal-9887.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9887.png,Normal/masks/Normal-9887.png,True,False,299,299,L,134.73,4221d738d922ea5a138deb956f6905ee
Normal-9888.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9888.png,Normal/images/Normal-9888.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9888.png,Normal/masks/Normal-9888.png,True,False,299,299,L,103.96,08b886ad5cb5e00af1bc62c6c94ba353
Normal-9889.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9889.png,Normal/images/Normal-9889.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9889.png,Normal/masks/Normal-9889.png,True,False,299,299,L,182.13,56ba159a2b48fd8b35025eb260e8a10a
Normal-989.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-989.png,Normal/images/Normal-989.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-989.png,Normal/masks/Normal-989.png,True,False,299,299,L,113.0,f78d255ecfd43fa29dae8ddcb4fb6453
Normal-9890.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9890.png,Normal/images/Normal-9890.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9890.png,Normal/masks/Normal-9890.png,True,False,299,299,L,125.51,b86b8686e2f72defef056c755f65f2f3
Normal-9891.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9891.png,Normal/images/Normal-9891.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9891.png,Normal/masks/Normal-9891.png,True,False,299,299,L,141.08,73964310fcffba32eaaa5c6aa016c2df
Normal-9892.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9892.png,Normal/images/Normal-9892.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9892.png,Normal/masks/Normal-9892.png,True,False,299,299,L,115.04,5014995546df3796ef6a134bc8f4f5d0
Normal-9893.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9893.png,Normal/images/Normal-9893.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9893.png,Normal/masks/Normal-9893.png,True,False,299,299,L,119.57,5533babec5aba3afb7d6336f3f4a1c53
Normal-9894.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9894.png,Normal/images/Normal-9894.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9894.png,Normal/masks/Normal-9894.png,True,False,299,299,L,123.77,9245ee875401de1f81b03cd24b5ca08b
Normal-9895.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9895.png,Normal/images/Normal-9895.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9895.png,Normal/masks/Normal-9895.png,True,False,299,299,L,131.38,ef26dbefe26f2744c42535dc18e97a7c
Normal-9896.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9896.png,Normal/images/Normal-9896.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9896.png,Normal/masks/Normal-9896.png,True,False,299,299,L,117.3,72c5ca61a4b7cbe601744800a13e95b1
Normal-9897.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9897.png,Normal/images/Normal-9897.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9897.png,Normal/masks/Normal-9897.png,True,False,299,299,L,158.67,95357dbe695f623d809a5b71b4fa6c21
Normal-9898.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9898.png,Normal/images/Normal-9898.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9898.png,Normal/masks/Normal-9898.png,True,False,299,299,L,110.43,d459278caf6254fb064e487f4b2acfa1
Normal-9899.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9899.png,Normal/images/Normal-9899.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9899.png,Normal/masks/Normal-9899.png,True,False,299,299,L,92.82,73060f5d8e58112ff79661086f0603ac
Normal-99.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-99.png,Normal/images/Normal-99.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-99.png,Normal/masks/Normal-99.png,True,False,299,299,L,129.66,71a7a5d3467aa21868e6a906478ec8ee
Normal-990.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-990.png,Normal/images/Normal-990.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-990.png,Normal/masks/Normal-990.png,True,False,299,299,L,132.12,0c65dad8e1838c43d5a29ebdafaddeca
Normal-9900.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9900.png,Normal/images/Normal-9900.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9900.png,Normal/masks/Normal-9900.png,True,False,299,299,L,86.41,44ea1a658d02fc4fd15d6df72085a990
Normal-9901.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9901.png,Normal/images/Normal-9901.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9901.png,Normal/masks/Normal-9901.png,True,False,299,299,L,163.94,7b6ed0755d4058aaa524bbe2e22069c4
Normal-9902.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9902.png,Normal/images/Normal-9902.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9902.png,Normal/masks/Normal-9902.png,True,False,299,299,L,166.88,3d3d2e6d041f49969921b298a9ca1f93
Normal-9903.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9903.png,Normal/images/Normal-9903.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9903.png,Normal/masks/Normal-9903.png,True,False,299,299,L,120.11,09b35998dc70e67168ff53b25a041eb2
Normal-9904.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9904.png,Normal/images/Normal-9904.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9904.png,Normal/masks/Normal-9904.png,True,False,299,299,L,124.04,6475315bdb7836eeb97ada523b84ae07
Normal-9905.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9905.png,Normal/images/Normal-9905.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9905.png,Normal/masks/Normal-9905.png,True,False,299,299,L,158.01,43b6088af4e18ff87207ee02f7ba7081
Normal-9906.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9906.png,Normal/images/Normal-9906.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9906.png,Normal/masks/Normal-9906.png,True,False,299,299,L,140.2,10d24520f0e0ee55faa03f665042d000
Normal-9907.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9907.png,Normal/images/Normal-9907.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9907.png,Normal/masks/Normal-9907.png,True,False,299,299,L,169.51,2ecfc274f11b8fb995fef230a1b784af
Normal-9908.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9908.png,Normal/images/Normal-9908.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9908.png,Normal/masks/Normal-9908.png,True,False,299,299,L,137.06,8fc6e117e1236d79fc0330df28a8bc9d
Normal-9909.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9909.png,Normal/images/Normal-9909.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9909.png,Normal/masks/Normal-9909.png,True,False,299,299,L,162.57,8ef5a95457b06632ac11590ec521d8fa
Normal-991.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-991.png,Normal/images/Normal-991.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-991.png,Normal/masks/Normal-991.png,True,False,299,299,L,105.05,6d2f7253e997fdcbaa82a4af1df13753
Normal-9910.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9910.png,Normal/images/Normal-9910.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9910.png,Normal/masks/Normal-9910.png,True,False,299,299,L,126.64,1dd1cb34f70f850bd1b97c3af3d1174e
Normal-9911.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9911.png,Normal/images/Normal-9911.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9911.png,Normal/masks/Normal-9911.png,True,False,299,299,L,133.81,9939fad481d965eebb34a11349acd7f1
Normal-9912.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9912.png,Normal/images/Normal-9912.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9912.png,Normal/masks/Normal-9912.png,True,False,299,299,L,184.74,de01c532d4e2eb16477e5da3baebccbc
Normal-9913.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9913.png,Normal/images/Normal-9913.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9913.png,Normal/masks/Normal-9913.png,True,False,299,299,L,179.49,1dfd1d38ebf223910b2e7d0d957451d4
Normal-9914.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9914.png,Normal/images/Normal-9914.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9914.png,Normal/masks/Normal-9914.png,True,False,299,299,L,126.79,52f0dc6df68e6885708266699a331d21
Normal-9915.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9915.png,Normal/images/Normal-9915.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9915.png,Normal/masks/Normal-9915.png,True,False,299,299,L,99.83,e2d7d57d787861637e41b29b9f72b063
Normal-9916.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9916.png,Normal/images/Normal-9916.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9916.png,Normal/masks/Normal-9916.png,True,False,299,299,L,139.29,1df4e759d0a2769f7dc024c963adf180
Normal-9917.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9917.png,Normal/images/Normal-9917.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9917.png,Normal/masks/Normal-9917.png,True,False,299,299,L,161.75,334da62ce2102f1359eb86c55616b81e
Normal-9918.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9918.png,Normal/images/Normal-9918.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9918.png,Normal/masks/Normal-9918.png,True,False,299,299,L,96.37,f6a87c4cc0796db7fd015649b9eacefb
Normal-9919.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9919.png,Normal/images/Normal-9919.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9919.png,Normal/masks/Normal-9919.png,True,False,299,299,L,136.37,17376c93298547c97b74731cc9c2212b
Normal-992.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-992.png,Normal/images/Normal-992.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-992.png,Normal/masks/Normal-992.png,True,False,299,299,L,113.67,55ad7a5c7c29579ad5874b65d7b8b079
Normal-9920.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9920.png,Normal/images/Normal-9920.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9920.png,Normal/masks/Normal-9920.png,True,False,299,299,L,132.95,88b9977fb0d6093c3a1048c29578cbe4
Normal-9921.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9921.png,Normal/images/Normal-9921.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9921.png,Normal/masks/Normal-9921.png,True,False,299,299,L,104.01,4aa556c6d902d46d1a6cad41919bd3fb
Normal-9922.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9922.png,Normal/images/Normal-9922.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9922.png,Normal/masks/Normal-9922.png,True,False,299,299,L,133.03,f46b0accff96465e6e580930203df82b
Normal-9923.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9923.png,Normal/images/Normal-9923.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9923.png,Normal/masks/Normal-9923.png,True,False,299,299,L,139.6,2f2891f77d48f17e7c4e8e439b02d228
Normal-9924.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9924.png,Normal/images/Normal-9924.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9924.png,Normal/masks/Normal-9924.png,True,False,299,299,L,157.96,841f08dee3c82f0e883b3b876d55470b
Normal-9925.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9925.png,Normal/images/Normal-9925.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9925.png,Normal/masks/Normal-9925.png,True,False,299,299,L,112.28,552426b351bbaadc03efb58c16e2be5f
Normal-9926.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9926.png,Normal/images/Normal-9926.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9926.png,Normal/masks/Normal-9926.png,True,False,299,299,L,126.79,291a5373b0cb36647b6002f1fa4ed09a
Normal-9927.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9927.png,Normal/images/Normal-9927.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9927.png,Normal/masks/Normal-9927.png,True,False,299,299,L,120.06,8b0258f9ad72d8a305976f1daf7dbe69
Normal-9928.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9928.png,Normal/images/Normal-9928.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9928.png,Normal/masks/Normal-9928.png,True,False,299,299,L,159.84,0e77deeaaa4e67966be891f55bf9640c
Normal-9929.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9929.png,Normal/images/Normal-9929.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9929.png,Normal/masks/Normal-9929.png,True,False,299,299,L,108.74,ed9e992da1bdf4f2b9eea071bbec8c7b
Normal-993.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-993.png,Normal/images/Normal-993.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-993.png,Normal/masks/Normal-993.png,True,False,299,299,L,122.08,7e18d541f79eba92f15a7dde0fa3fd87
Normal-9930.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9930.png,Normal/images/Normal-9930.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9930.png,Normal/masks/Normal-9930.png,True,False,299,299,L,135.15,6ebe3feb7d43b126155939ea9bd17606
Normal-9931.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9931.png,Normal/images/Normal-9931.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9931.png,Normal/masks/Normal-9931.png,True,False,299,299,L,120.82,ef3c3584d0de8499702be4909dbff122
Normal-9932.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9932.png,Normal/images/Normal-9932.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9932.png,Normal/masks/Normal-9932.png,True,False,299,299,L,183.36,0d8d4883818b9f5882eeb79e65a4367c
Normal-9933.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9933.png,Normal/images/Normal-9933.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9933.png,Normal/masks/Normal-9933.png,True,False,299,299,L,135.8,ebd6bf7803b7aaf21ce0314f8131a5e8
Normal-9934.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9934.png,Normal/images/Normal-9934.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9934.png,Normal/masks/Normal-9934.png,True,False,299,299,L,161.88,15f902e5b721dd913104b576fbb9d400
Normal-9935.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9935.png,Normal/images/Normal-9935.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9935.png,Normal/masks/Normal-9935.png,True,False,299,299,L,153.02,1410c5a80770b4e176b0c252d353d0f5
Normal-9936.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9936.png,Normal/images/Normal-9936.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9936.png,Normal/masks/Normal-9936.png,True,False,299,299,L,144.35,1688594cb8cc7a351b3e9ce95f0a42f5
Normal-9937.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9937.png,Normal/images/Normal-9937.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9937.png,Normal/masks/Normal-9937.png,True,False,299,299,L,170.37,3f55d435da3132096540e00f49bfde8f
Normal-9938.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9938.png,Normal/images/Normal-9938.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9938.png,Normal/masks/Normal-9938.png,True,False,299,299,L,165.55,4ff69b7f3e41367ad41016f4e6b7f689
Normal-9939.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9939.png,Normal/images/Normal-9939.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9939.png,Normal/masks/Normal-9939.png,True,False,299,299,L,142.88,51253ae22ada13d9d87496ff56ad15a4
Normal-994.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-994.png,Normal/images/Normal-994.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-994.png,Normal/masks/Normal-994.png,True,False,299,299,L,116.75,c996245e858ef86826e426f35cec8b54
Normal-9940.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9940.png,Normal/images/Normal-9940.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9940.png,Normal/masks/Normal-9940.png,True,False,299,299,L,131.81,b051c4ee86ba2c72fbf96c5c11c6ac61
Normal-9941.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9941.png,Normal/images/Normal-9941.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9941.png,Normal/masks/Normal-9941.png,True,False,299,299,L,146.44,40a04027ca1100d04d4c762be931e45f
Normal-9942.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9942.png,Normal/images/Normal-9942.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9942.png,Normal/masks/Normal-9942.png,True,False,299,299,L,156.56,cd97d288b7b93c4c89fa29768c11488b
Normal-9943.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9943.png,Normal/images/Normal-9943.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9943.png,Normal/masks/Normal-9943.png,True,False,299,299,L,125.98,066422a3a4c5d7f97f73161aa50d7379
Normal-9944.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9944.png,Normal/images/Normal-9944.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9944.png,Normal/masks/Normal-9944.png,True,False,299,299,L,151.74,b977896e3a8a5f59ff5a3ff0d318b2bd
Normal-9945.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9945.png,Normal/images/Normal-9945.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9945.png,Normal/masks/Normal-9945.png,True,False,299,299,L,140.81,f5ffc3a61a2931f5c7af858d5ff246bb
Normal-9946.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9946.png,Normal/images/Normal-9946.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9946.png,Normal/masks/Normal-9946.png,True,False,299,299,L,134.55,1522a8207fddcdf9cd9c744fd9e9aef1
Normal-9947.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9947.png,Normal/images/Normal-9947.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9947.png,Normal/masks/Normal-9947.png,True,False,299,299,L,106.28,a8f5015827cc5c1f20a417167763f6eb
Normal-9948.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9948.png,Normal/images/Normal-9948.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9948.png,Normal/masks/Normal-9948.png,True,False,299,299,L,163.3,dc3be3fc6cf7f1bb6f1d8d0fb0b678a4
Normal-9949.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9949.png,Normal/images/Normal-9949.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9949.png,Normal/masks/Normal-9949.png,True,False,299,299,L,103.73,d0fbf467d3f776bd6b697375604f5ea0
Normal-995.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-995.png,Normal/images/Normal-995.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-995.png,Normal/masks/Normal-995.png,True,False,299,299,L,143.92,ed95d33df2f11bbc434fe11c69046222
Normal-9950.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9950.png,Normal/images/Normal-9950.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9950.png,Normal/masks/Normal-9950.png,True,False,299,299,L,133.36,3a4900558f3f197e44d40a0bf0b91e56
Normal-9951.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9951.png,Normal/images/Normal-9951.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9951.png,Normal/masks/Normal-9951.png,True,False,299,299,L,109.99,573310c4f42f4bb99a1567a3b81f896c
Normal-9952.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9952.png,Normal/images/Normal-9952.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9952.png,Normal/masks/Normal-9952.png,True,False,299,299,L,145.38,0b26ae8a787b5a7358ebb90d801016c8
Normal-9953.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9953.png,Normal/images/Normal-9953.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9953.png,Normal/masks/Normal-9953.png,True,False,299,299,L,121.15,a69c823a4c9a329820b4263fc8bab904
Normal-9954.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9954.png,Normal/images/Normal-9954.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9954.png,Normal/masks/Normal-9954.png,True,False,299,299,L,167.04,217b9af74d621be73f24baaf1bcef889
Normal-9955.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9955.png,Normal/images/Normal-9955.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9955.png,Normal/masks/Normal-9955.png,True,False,299,299,L,164.48,cdf75f4b478cf7be20f8b82ba7c28cf5
Normal-9956.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9956.png,Normal/images/Normal-9956.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9956.png,Normal/masks/Normal-9956.png,True,False,299,299,L,170.37,045bce6c646f1eebbc7ae4cc669e85d8
Normal-9957.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9957.png,Normal/images/Normal-9957.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9957.png,Normal/masks/Normal-9957.png,True,False,299,299,L,109.14,6e73ae41742e2f1ffaf45eebf70f5389
Normal-9958.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9958.png,Normal/images/Normal-9958.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9958.png,Normal/masks/Normal-9958.png,True,False,299,299,L,123.41,a9280d03928d0b74fe2f21beafd2bee5
Normal-9959.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9959.png,Normal/images/Normal-9959.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9959.png,Normal/masks/Normal-9959.png,True,False,299,299,L,155.76,955e5decd4006d7a3685fd1bbd38520d
Normal-996.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-996.png,Normal/images/Normal-996.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-996.png,Normal/masks/Normal-996.png,True,False,299,299,L,113.54,fad6cdfb2880f312f8875dc1b06d5635
Normal-9960.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9960.png,Normal/images/Normal-9960.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9960.png,Normal/masks/Normal-9960.png,True,False,299,299,L,115.42,282520b4399fa9503da50c4d13910000
Normal-9961.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9961.png,Normal/images/Normal-9961.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9961.png,Normal/masks/Normal-9961.png,True,False,299,299,L,161.33,da9e64b121e497c8ad750dfc9b8cb4a9
Normal-9962.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9962.png,Normal/images/Normal-9962.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9962.png,Normal/masks/Normal-9962.png,True,False,299,299,L,132.38,99bf00e1a3402fe865a6ed850af6b33c
Normal-9963.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9963.png,Normal/images/Normal-9963.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9963.png,Normal/masks/Normal-9963.png,True,False,299,299,L,155.64,ea78c63f231c33a2be68ed80a660e708
Normal-9964.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9964.png,Normal/images/Normal-9964.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9964.png,Normal/masks/Normal-9964.png,True,False,299,299,L,166.32,9e8c3f8e2237b32a04e3b569f3e5dcd1
Normal-9965.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9965.png,Normal/images/Normal-9965.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9965.png,Normal/masks/Normal-9965.png,True,False,299,299,L,137.48,f494951521d39b48697b718b2def3ee4
Normal-9966.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9966.png,Normal/images/Normal-9966.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9966.png,Normal/masks/Normal-9966.png,True,False,299,299,L,147.56,31f1b830fce521750b01a9390234cb4b
Normal-9967.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9967.png,Normal/images/Normal-9967.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9967.png,Normal/masks/Normal-9967.png,True,False,299,299,L,186.04,a658b9881d0bd95679f2a94962dea2e6
Normal-9968.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9968.png,Normal/images/Normal-9968.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9968.png,Normal/masks/Normal-9968.png,True,False,299,299,L,89.05,085bd901a1e2a69c385b230ca7e6673b
Normal-9969.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9969.png,Normal/images/Normal-9969.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9969.png,Normal/masks/Normal-9969.png,True,False,299,299,L,104.8,0be468fba58819ecea8969aba40a13b5
Normal-997.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-997.png,Normal/images/Normal-997.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-997.png,Normal/masks/Normal-997.png,True,False,299,299,L,114.66,d124173d4fcdf037c9851efc727207a9
Normal-9970.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9970.png,Normal/images/Normal-9970.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9970.png,Normal/masks/Normal-9970.png,True,False,299,299,L,147.04,28bf900b32eff54925b13b4179a0dc46
Normal-9971.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9971.png,Normal/images/Normal-9971.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9971.png,Normal/masks/Normal-9971.png,True,False,299,299,L,131.19,7e14f94eb9cb25aef6a289f6f57c764f
Normal-9972.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9972.png,Normal/images/Normal-9972.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9972.png,Normal/masks/Normal-9972.png,True,False,299,299,L,111.15,5e61a41bde178725d45c2896b347036a
Normal-9973.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9973.png,Normal/images/Normal-9973.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9973.png,Normal/masks/Normal-9973.png,True,False,299,299,L,83.27,e72aa12b48a39d034a8517f7a8e648a6
Normal-9974.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9974.png,Normal/images/Normal-9974.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9974.png,Normal/masks/Normal-9974.png,True,False,299,299,L,119.6,ac18475f48f4ae6c30d7e6a3a7ce0f0a
Normal-9975.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9975.png,Normal/images/Normal-9975.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9975.png,Normal/masks/Normal-9975.png,True,False,299,299,L,168.4,4081ad6d84c3099443c034037288108c
Normal-9976.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9976.png,Normal/images/Normal-9976.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9976.png,Normal/masks/Normal-9976.png,True,False,299,299,L,117.23,8fd9fd0a7ae8b13f180d384bf848c83b
Normal-9977.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9977.png,Normal/images/Normal-9977.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9977.png,Normal/masks/Normal-9977.png,True,False,299,299,L,173.84,2380291d35f5592dc7627ad676a5cc80
Normal-9978.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9978.png,Normal/images/Normal-9978.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9978.png,Normal/masks/Normal-9978.png,True,False,299,299,L,143.41,99f869e89e214508198e89f301a9d29b
Normal-9979.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9979.png,Normal/images/Normal-9979.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9979.png,Normal/masks/Normal-9979.png,True,False,299,299,L,159.62,99fafb59c195d496f932715287c93272
Normal-998.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-998.png,Normal/images/Normal-998.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-998.png,Normal/masks/Normal-998.png,True,False,299,299,L,132.54,b0610aefa1fbf40cbe700e8dd55bf3dc
Normal-9980.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9980.png,Normal/images/Normal-9980.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9980.png,Normal/masks/Normal-9980.png,True,False,299,299,L,121.11,e5417f85b666a270d1938be870cf6403
Normal-9981.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9981.png,Normal/images/Normal-9981.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9981.png,Normal/masks/Normal-9981.png,True,False,299,299,L,115.97,e921894b02f72d3a644ba5b051e0c700
Normal-9982.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9982.png,Normal/images/Normal-9982.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9982.png,Normal/masks/Normal-9982.png,True,False,299,299,L,127.84,f040399a47d105ca103b2e8a4204cefe
Normal-9983.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9983.png,Normal/images/Normal-9983.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9983.png,Normal/masks/Normal-9983.png,True,False,299,299,L,138.65,cfbe87b0fc6ee4bc846689acec1f6af2
Normal-9984.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9984.png,Normal/images/Normal-9984.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9984.png,Normal/masks/Normal-9984.png,True,False,299,299,L,168.05,f7e7b537a26d29915e33db529f4e50a9
Normal-9985.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9985.png,Normal/images/Normal-9985.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9985.png,Normal/masks/Normal-9985.png,True,False,299,299,L,104.84,186d3034a4deebff8c32657a13dbb083
Normal-9986.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9986.png,Normal/images/Normal-9986.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9986.png,Normal/masks/Normal-9986.png,True,False,299,299,L,168.36,014d689d8d3010d8a9ad1b6c8c933d74
Normal-9987.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9987.png,Normal/images/Normal-9987.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9987.png,Normal/masks/Normal-9987.png,True,False,299,299,L,102.66,bbb8e1917036235dfdfeca3a1e529b74
Normal-9988.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9988.png,Normal/images/Normal-9988.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9988.png,Normal/masks/Normal-9988.png,True,False,299,299,L,118.02,9302c3df575816d5a2a252bd28886fa6
Normal-9989.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9989.png,Normal/images/Normal-9989.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9989.png,Normal/masks/Normal-9989.png,True,False,299,299,L,144.07,8e081ca89c40f91fe97720f7a24a85fc
Normal-999.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-999.png,Normal/images/Normal-999.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-999.png,Normal/masks/Normal-999.png,True,False,299,299,L,123.05,e6656225fc982a561ac1d6836186c0ef
Normal-9990.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9990.png,Normal/images/Normal-9990.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9990.png,Normal/masks/Normal-9990.png,True,False,299,299,L,126.7,2b24b7ef1770137fde7a0558ff0b571a
Normal-9991.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9991.png,Normal/images/Normal-9991.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9991.png,Normal/masks/Normal-9991.png,True,False,299,299,L,100.64,27135aa605fef781b438ec4db83473d9
Normal-9992.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9992.png,Normal/images/Normal-9992.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9992.png,Normal/masks/Normal-9992.png,True,False,299,299,L,138.85,2b4367bfecae12115a2dc7b19b463a37
Normal-9993.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9993.png,Normal/images/Normal-9993.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9993.png,Normal/masks/Normal-9993.png,True,False,299,299,L,112.71,3b10de02cfbe1c55eadaf15e6ce70930
Normal-9994.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9994.png,Normal/images/Normal-9994.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9994.png,Normal/masks/Normal-9994.png,True,False,299,299,L,101.34,f5fb9687c1c9c0bf340387593f321b29
Normal-9995.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9995.png,Normal/images/Normal-9995.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9995.png,Normal/masks/Normal-9995.png,True,False,299,299,L,120.04,32b01c219cb87a0674c39fadc3273578
Normal-9996.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9996.png,Normal/images/Normal-9996.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9996.png,Normal/masks/Normal-9996.png,True,False,299,299,L,134.41,32cc8260546af740b929dd9b1002ced4
Normal-9997.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9997.png,Normal/images/Normal-9997.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9997.png,Normal/masks/Normal-9997.png,True,False,299,299,L,129.85,f0e959af6b5ae37b02f8bc3d2db661a0
Normal-9998.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9998.png,Normal/images/Normal-9998.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9998.png,Normal/masks/Normal-9998.png,True,False,299,299,L,156.8,d3d12bc3b86f60df5f56526496dbc1fd
Normal-9999.png,Normal,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images/Normal-9999.png,Normal/images/Normal-9999.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/masks/Normal-9999.png,Normal/masks/Normal-9999.png,True,False,299,299,L,128.14,67105a8c76ff1fb45d6a13cc1b5f7fff
Viral Pneumonia-1.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1.png,Viral Pneumonia/images/Viral Pneumonia-1.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1.png,Viral Pneumonia/masks/Viral Pneumonia-1.png,True,False,299,299,L,121.5,3158b8fe7878ae73f8718b55ded7fff2
Viral Pneumonia-10.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-10.png,Viral Pneumonia/images/Viral Pneumonia-10.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-10.png,Viral Pneumonia/masks/Viral Pneumonia-10.png,True,False,299,299,L,132.67,87593be64c4d400e28ff9b33853631a3
Viral Pneumonia-100.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-100.png,Viral Pneumonia/images/Viral Pneumonia-100.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-100.png,Viral Pneumonia/masks/Viral Pneumonia-100.png,True,False,299,299,L,121.97,9f17f132aff79d1361900626e4db9272
Viral Pneumonia-1000.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1000.png,Viral Pneumonia/images/Viral Pneumonia-1000.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1000.png,Viral Pneumonia/masks/Viral Pneumonia-1000.png,True,False,299,299,L,122.94,66205992e18c051d728f76383cad4da0
Viral Pneumonia-1001.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1001.png,Viral Pneumonia/images/Viral Pneumonia-1001.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1001.png,Viral Pneumonia/masks/Viral Pneumonia-1001.png,True,False,299,299,L,166.23,df72fce3ca7b130ab6ea6e531d8902b3
Viral Pneumonia-1002.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1002.png,Viral Pneumonia/images/Viral Pneumonia-1002.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1002.png,Viral Pneumonia/masks/Viral Pneumonia-1002.png,True,False,299,299,L,115.76,7350fef3ccd26c15ad76f54f8ea6309b
Viral Pneumonia-1003.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1003.png,Viral Pneumonia/images/Viral Pneumonia-1003.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1003.png,Viral Pneumonia/masks/Viral Pneumonia-1003.png,True,False,299,299,L,121.9,b76684ca49727b3faccc74e508aac7de
Viral Pneumonia-1004.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1004.png,Viral Pneumonia/images/Viral Pneumonia-1004.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1004.png,Viral Pneumonia/masks/Viral Pneumonia-1004.png,True,False,299,299,RGB,119.68,11bc273ec28153e0cc08568ed2f8349e
Viral Pneumonia-1005.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1005.png,Viral Pneumonia/images/Viral Pneumonia-1005.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1005.png,Viral Pneumonia/masks/Viral Pneumonia-1005.png,True,False,299,299,RGB,134.43,845ba4ac50c2a586345fb2bb3ab97565
Viral Pneumonia-1006.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1006.png,Viral Pneumonia/images/Viral Pneumonia-1006.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1006.png,Viral Pneumonia/masks/Viral Pneumonia-1006.png,True,False,299,299,L,113.4,d507991581713650c16e11eb9c93091d
Viral Pneumonia-1007.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1007.png,Viral Pneumonia/images/Viral Pneumonia-1007.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1007.png,Viral Pneumonia/masks/Viral Pneumonia-1007.png,True,False,299,299,L,163.14,bf818f598348b31db221b988a3b414f3
Viral Pneumonia-1008.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1008.png,Viral Pneumonia/images/Viral Pneumonia-1008.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1008.png,Viral Pneumonia/masks/Viral Pneumonia-1008.png,True,False,299,299,L,127.1,7acf6f6c1d0fce0106d9149fe8802269
Viral Pneumonia-1009.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1009.png,Viral Pneumonia/images/Viral Pneumonia-1009.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1009.png,Viral Pneumonia/masks/Viral Pneumonia-1009.png,True,False,299,299,L,116.13,6358aac29071efb2993d2290fd839b82
Viral Pneumonia-101.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-101.png,Viral Pneumonia/images/Viral Pneumonia-101.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-101.png,Viral Pneumonia/masks/Viral Pneumonia-101.png,True,False,299,299,L,128.05,8c92c2caa58430cc06887582ed973d27
Viral Pneumonia-1010.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1010.png,Viral Pneumonia/images/Viral Pneumonia-1010.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1010.png,Viral Pneumonia/masks/Viral Pneumonia-1010.png,True,False,299,299,L,166.72,9e3dafb5dc7e16f49ff50b792b1b3973
Viral Pneumonia-1011.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1011.png,Viral Pneumonia/images/Viral Pneumonia-1011.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1011.png,Viral Pneumonia/masks/Viral Pneumonia-1011.png,True,False,299,299,L,145.58,d36f1e3567a27c75cd2138e6c6ec5229
Viral Pneumonia-1012.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1012.png,Viral Pneumonia/images/Viral Pneumonia-1012.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1012.png,Viral Pneumonia/masks/Viral Pneumonia-1012.png,True,False,299,299,L,91.39,8c43e8218890e1c9b8e464c21803686a
Viral Pneumonia-1013.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1013.png,Viral Pneumonia/images/Viral Pneumonia-1013.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1013.png,Viral Pneumonia/masks/Viral Pneumonia-1013.png,True,False,299,299,L,132.75,3f07825623faa13398e0f7043b9ed970
Viral Pneumonia-1014.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1014.png,Viral Pneumonia/images/Viral Pneumonia-1014.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1014.png,Viral Pneumonia/masks/Viral Pneumonia-1014.png,True,False,299,299,L,151.55,a95c40201039ee8561cbfd41ca36e20c
Viral Pneumonia-1015.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1015.png,Viral Pneumonia/images/Viral Pneumonia-1015.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1015.png,Viral Pneumonia/masks/Viral Pneumonia-1015.png,True,False,299,299,L,94.03,8f576a8ed2a4de17b5f7b431b62f444f
Viral Pneumonia-1016.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1016.png,Viral Pneumonia/images/Viral Pneumonia-1016.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1016.png,Viral Pneumonia/masks/Viral Pneumonia-1016.png,True,False,299,299,L,127.3,81ff4145c49b9e4fa0d88d1d018dd3ca
Viral Pneumonia-1017.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1017.png,Viral Pneumonia/images/Viral Pneumonia-1017.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1017.png,Viral Pneumonia/masks/Viral Pneumonia-1017.png,True,False,299,299,RGB,142.98,9595f522b5c0411415fcefbb6e4cbaec
Viral Pneumonia-1018.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1018.png,Viral Pneumonia/images/Viral Pneumonia-1018.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1018.png,Viral Pneumonia/masks/Viral Pneumonia-1018.png,True,False,299,299,L,128.58,c034dd88fe1db3c8e22e27ab1ed4105a
Viral Pneumonia-1019.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1019.png,Viral Pneumonia/images/Viral Pneumonia-1019.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1019.png,Viral Pneumonia/masks/Viral Pneumonia-1019.png,True,False,299,299,L,113.35,01cab5f49c1f9044c301519fee5cb1fb
Viral Pneumonia-102.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-102.png,Viral Pneumonia/images/Viral Pneumonia-102.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-102.png,Viral Pneumonia/masks/Viral Pneumonia-102.png,True,False,299,299,L,119.73,4f905c39a62ad9834244eaf884237da0
Viral Pneumonia-1020.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1020.png,Viral Pneumonia/images/Viral Pneumonia-1020.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1020.png,Viral Pneumonia/masks/Viral Pneumonia-1020.png,True,False,299,299,L,113.12,9d4274069c180e17aba2d5d551c192c9
Viral Pneumonia-1021.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1021.png,Viral Pneumonia/images/Viral Pneumonia-1021.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1021.png,Viral Pneumonia/masks/Viral Pneumonia-1021.png,True,False,299,299,L,107.01,cb9e45258f993b0519fe88c4175d140d
Viral Pneumonia-1022.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1022.png,Viral Pneumonia/images/Viral Pneumonia-1022.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1022.png,Viral Pneumonia/masks/Viral Pneumonia-1022.png,True,False,299,299,L,135.26,0fa156941a022f51541880451a0f425d
Viral Pneumonia-1023.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1023.png,Viral Pneumonia/images/Viral Pneumonia-1023.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1023.png,Viral Pneumonia/masks/Viral Pneumonia-1023.png,True,False,299,299,L,138.43,457a55a1a37f99812fc2ec8dd30785f8
Viral Pneumonia-1024.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1024.png,Viral Pneumonia/images/Viral Pneumonia-1024.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1024.png,Viral Pneumonia/masks/Viral Pneumonia-1024.png,True,False,299,299,L,115.36,81e2a42635a8d689547fdc97244d5357
Viral Pneumonia-1025.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1025.png,Viral Pneumonia/images/Viral Pneumonia-1025.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1025.png,Viral Pneumonia/masks/Viral Pneumonia-1025.png,True,False,299,299,L,143.59,31fbc123a2e94ffd8b8acd4ded512a2c
Viral Pneumonia-1026.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1026.png,Viral Pneumonia/images/Viral Pneumonia-1026.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1026.png,Viral Pneumonia/masks/Viral Pneumonia-1026.png,True,False,299,299,L,152.1,f0f9e1d0e4b850b7f1a9b2391c15906f
Viral Pneumonia-1027.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1027.png,Viral Pneumonia/images/Viral Pneumonia-1027.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1027.png,Viral Pneumonia/masks/Viral Pneumonia-1027.png,True,False,299,299,L,155.12,1e55344480985e5df29f1e8ff7319e13
Viral Pneumonia-1028.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1028.png,Viral Pneumonia/images/Viral Pneumonia-1028.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1028.png,Viral Pneumonia/masks/Viral Pneumonia-1028.png,True,False,299,299,L,115.19,0ffb052febe50ae01f2da4a67e0d28c2
Viral Pneumonia-1029.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1029.png,Viral Pneumonia/images/Viral Pneumonia-1029.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1029.png,Viral Pneumonia/masks/Viral Pneumonia-1029.png,True,False,299,299,L,132.26,099fff40049880ac4f3da979ccfedf50
Viral Pneumonia-103.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-103.png,Viral Pneumonia/images/Viral Pneumonia-103.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-103.png,Viral Pneumonia/masks/Viral Pneumonia-103.png,True,False,299,299,L,139.54,ae11371b771d323d072cc68df0f3433d
Viral Pneumonia-1030.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1030.png,Viral Pneumonia/images/Viral Pneumonia-1030.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1030.png,Viral Pneumonia/masks/Viral Pneumonia-1030.png,True,False,299,299,L,125.83,a7266457d18aaf315b14081e013e51c2
Viral Pneumonia-1031.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1031.png,Viral Pneumonia/images/Viral Pneumonia-1031.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1031.png,Viral Pneumonia/masks/Viral Pneumonia-1031.png,True,False,299,299,L,141.48,3bfc9b2373f6286cc63f26225647b4c1
Viral Pneumonia-1032.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1032.png,Viral Pneumonia/images/Viral Pneumonia-1032.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1032.png,Viral Pneumonia/masks/Viral Pneumonia-1032.png,True,False,299,299,L,150.3,ff4ea10b61db3cc3c0956498bf74f7be
Viral Pneumonia-1033.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1033.png,Viral Pneumonia/images/Viral Pneumonia-1033.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1033.png,Viral Pneumonia/masks/Viral Pneumonia-1033.png,True,False,299,299,L,126.49,0b46b0143eb0a668f4eda01b2ecd804c
Viral Pneumonia-1034.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1034.png,Viral Pneumonia/images/Viral Pneumonia-1034.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1034.png,Viral Pneumonia/masks/Viral Pneumonia-1034.png,True,False,299,299,L,127.64,d5395817602d4ca74d1c0803bc2ce94b
Viral Pneumonia-1035.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1035.png,Viral Pneumonia/images/Viral Pneumonia-1035.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1035.png,Viral Pneumonia/masks/Viral Pneumonia-1035.png,True,False,299,299,L,156.63,d13ae516c9452ef73cfe1340ee77bdda
Viral Pneumonia-1036.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1036.png,Viral Pneumonia/images/Viral Pneumonia-1036.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1036.png,Viral Pneumonia/masks/Viral Pneumonia-1036.png,True,False,299,299,L,91.29,63970c776201359b0f14038c00ce8bb1
Viral Pneumonia-1037.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1037.png,Viral Pneumonia/images/Viral Pneumonia-1037.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1037.png,Viral Pneumonia/masks/Viral Pneumonia-1037.png,True,False,299,299,L,126.75,34c3a0ac968406d475b67a612f9423c1
Viral Pneumonia-1038.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1038.png,Viral Pneumonia/images/Viral Pneumonia-1038.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1038.png,Viral Pneumonia/masks/Viral Pneumonia-1038.png,True,False,299,299,L,99.73,818d0c6076db4f09d510aa3d2c9d7f39
Viral Pneumonia-1039.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1039.png,Viral Pneumonia/images/Viral Pneumonia-1039.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1039.png,Viral Pneumonia/masks/Viral Pneumonia-1039.png,True,False,299,299,L,141.54,35a3f744aa938e1c084bc138c41e1f2f
Viral Pneumonia-104.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-104.png,Viral Pneumonia/images/Viral Pneumonia-104.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-104.png,Viral Pneumonia/masks/Viral Pneumonia-104.png,True,False,299,299,L,113.32,8e6d98b9f726e86122871d9ffade2fe4
Viral Pneumonia-1040.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1040.png,Viral Pneumonia/images/Viral Pneumonia-1040.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1040.png,Viral Pneumonia/masks/Viral Pneumonia-1040.png,True,False,299,299,L,137.58,f9f2b2880133a60de13a198d6f47bc8e
Viral Pneumonia-1041.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1041.png,Viral Pneumonia/images/Viral Pneumonia-1041.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1041.png,Viral Pneumonia/masks/Viral Pneumonia-1041.png,True,False,299,299,L,134.97,8dedb5255d205020c3b9a03ea2d47171
Viral Pneumonia-1042.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1042.png,Viral Pneumonia/images/Viral Pneumonia-1042.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1042.png,Viral Pneumonia/masks/Viral Pneumonia-1042.png,True,False,299,299,L,120.08,052660d135baaa32983160d42555a856
Viral Pneumonia-1043.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1043.png,Viral Pneumonia/images/Viral Pneumonia-1043.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1043.png,Viral Pneumonia/masks/Viral Pneumonia-1043.png,True,False,299,299,L,143.04,b6ea18439de58e5458a56e39240f4947
Viral Pneumonia-1044.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1044.png,Viral Pneumonia/images/Viral Pneumonia-1044.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1044.png,Viral Pneumonia/masks/Viral Pneumonia-1044.png,True,False,299,299,L,122.85,ab5aaa6732e29fbf5424642bb2806ea0
Viral Pneumonia-1045.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1045.png,Viral Pneumonia/images/Viral Pneumonia-1045.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1045.png,Viral Pneumonia/masks/Viral Pneumonia-1045.png,True,False,299,299,L,113.4,117c05dc3459a294847f2a8ca813edc3
Viral Pneumonia-1046.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1046.png,Viral Pneumonia/images/Viral Pneumonia-1046.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1046.png,Viral Pneumonia/masks/Viral Pneumonia-1046.png,True,False,299,299,L,133.32,23c6dbfdc9859fbd06fc0d7343f3af7d
Viral Pneumonia-1047.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1047.png,Viral Pneumonia/images/Viral Pneumonia-1047.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1047.png,Viral Pneumonia/masks/Viral Pneumonia-1047.png,True,False,299,299,L,120.8,ec29991c2417b7838adfcd4f1be565a1
Viral Pneumonia-1048.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1048.png,Viral Pneumonia/images/Viral Pneumonia-1048.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1048.png,Viral Pneumonia/masks/Viral Pneumonia-1048.png,True,False,299,299,L,137.68,83691967ffc6a1dd1ff4d2637fe94926
Viral Pneumonia-1049.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1049.png,Viral Pneumonia/images/Viral Pneumonia-1049.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1049.png,Viral Pneumonia/masks/Viral Pneumonia-1049.png,True,False,299,299,L,133.39,7e76d4c970fbe8527be18a71a90ba79a
Viral Pneumonia-105.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-105.png,Viral Pneumonia/images/Viral Pneumonia-105.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-105.png,Viral Pneumonia/masks/Viral Pneumonia-105.png,True,False,299,299,L,158.25,1117a4a60686065f169be0e114928c76
Viral Pneumonia-1050.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1050.png,Viral Pneumonia/images/Viral Pneumonia-1050.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1050.png,Viral Pneumonia/masks/Viral Pneumonia-1050.png,True,False,299,299,L,121.93,ec4fc7ab1d5e2fabcf81c63bfb18d97a
Viral Pneumonia-1051.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1051.png,Viral Pneumonia/images/Viral Pneumonia-1051.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1051.png,Viral Pneumonia/masks/Viral Pneumonia-1051.png,True,False,299,299,L,175.21,283ba8eea34c18ee9ab56cb659d51c59
Viral Pneumonia-1052.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1052.png,Viral Pneumonia/images/Viral Pneumonia-1052.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1052.png,Viral Pneumonia/masks/Viral Pneumonia-1052.png,True,False,299,299,L,131.36,bea45adf9d89af380502199dcd6e230c
Viral Pneumonia-1053.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1053.png,Viral Pneumonia/images/Viral Pneumonia-1053.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1053.png,Viral Pneumonia/masks/Viral Pneumonia-1053.png,True,False,299,299,L,131.36,bea45adf9d89af380502199dcd6e230c
Viral Pneumonia-1054.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1054.png,Viral Pneumonia/images/Viral Pneumonia-1054.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1054.png,Viral Pneumonia/masks/Viral Pneumonia-1054.png,True,False,299,299,L,103.34,0f78de85e4cc2eef20016bbb98cb127b
Viral Pneumonia-1055.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1055.png,Viral Pneumonia/images/Viral Pneumonia-1055.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1055.png,Viral Pneumonia/masks/Viral Pneumonia-1055.png,True,False,299,299,L,125.19,130af026653d1b69d7e05ca1a9b9b7b3
Viral Pneumonia-1056.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1056.png,Viral Pneumonia/images/Viral Pneumonia-1056.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1056.png,Viral Pneumonia/masks/Viral Pneumonia-1056.png,True,False,299,299,L,112.42,c1ca0a5efae745bf041c37476c9203c6
Viral Pneumonia-1057.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1057.png,Viral Pneumonia/images/Viral Pneumonia-1057.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1057.png,Viral Pneumonia/masks/Viral Pneumonia-1057.png,True,False,299,299,L,123.37,c1564fc4fb2e7aee84c2c559e7f667ae
Viral Pneumonia-1058.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1058.png,Viral Pneumonia/images/Viral Pneumonia-1058.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1058.png,Viral Pneumonia/masks/Viral Pneumonia-1058.png,True,False,299,299,L,138.31,34d7c46419e23d685aedf09adef29493
Viral Pneumonia-1059.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1059.png,Viral Pneumonia/images/Viral Pneumonia-1059.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1059.png,Viral Pneumonia/masks/Viral Pneumonia-1059.png,True,False,299,299,L,125.28,fa82e8c3bd0df3fa25da55edade364f2
Viral Pneumonia-106.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-106.png,Viral Pneumonia/images/Viral Pneumonia-106.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-106.png,Viral Pneumonia/masks/Viral Pneumonia-106.png,True,False,299,299,L,135.35,fecbd4ac6f961dee7cf928fc30b12631
Viral Pneumonia-1060.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1060.png,Viral Pneumonia/images/Viral Pneumonia-1060.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1060.png,Viral Pneumonia/masks/Viral Pneumonia-1060.png,True,False,299,299,L,126.95,cd2d7a0412e0b3ddae28bde9957f9db7
Viral Pneumonia-1061.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1061.png,Viral Pneumonia/images/Viral Pneumonia-1061.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1061.png,Viral Pneumonia/masks/Viral Pneumonia-1061.png,True,False,299,299,L,143.77,9b73af695d2d1a1252fd3e3d29760d08
Viral Pneumonia-1062.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1062.png,Viral Pneumonia/images/Viral Pneumonia-1062.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1062.png,Viral Pneumonia/masks/Viral Pneumonia-1062.png,True,False,299,299,L,122.8,edbcb5c051f689c20f3fe714568fed4f
Viral Pneumonia-1063.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1063.png,Viral Pneumonia/images/Viral Pneumonia-1063.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1063.png,Viral Pneumonia/masks/Viral Pneumonia-1063.png,True,False,299,299,L,138.51,f4b9f25e5a9779b37dd59229dbdde2db
Viral Pneumonia-1064.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1064.png,Viral Pneumonia/images/Viral Pneumonia-1064.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1064.png,Viral Pneumonia/masks/Viral Pneumonia-1064.png,True,False,299,299,L,110.47,1b0c7421b1df08999c8e957608220bb9
Viral Pneumonia-1065.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1065.png,Viral Pneumonia/images/Viral Pneumonia-1065.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1065.png,Viral Pneumonia/masks/Viral Pneumonia-1065.png,True,False,299,299,L,107.99,5732d5c42ea4c49ca0bae680062308d6
Viral Pneumonia-1066.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1066.png,Viral Pneumonia/images/Viral Pneumonia-1066.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1066.png,Viral Pneumonia/masks/Viral Pneumonia-1066.png,True,False,299,299,L,144.5,fb5f2be2e7533554ecdcf92bb7a3d3de
Viral Pneumonia-1067.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1067.png,Viral Pneumonia/images/Viral Pneumonia-1067.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1067.png,Viral Pneumonia/masks/Viral Pneumonia-1067.png,True,False,299,299,L,129.68,65d40f8293a3e4fba059f33a9d1e9be3
Viral Pneumonia-1068.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1068.png,Viral Pneumonia/images/Viral Pneumonia-1068.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1068.png,Viral Pneumonia/masks/Viral Pneumonia-1068.png,True,False,299,299,L,135.78,57aecbbe0b06c5236d84bafc53648eaf
Viral Pneumonia-1069.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1069.png,Viral Pneumonia/images/Viral Pneumonia-1069.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1069.png,Viral Pneumonia/masks/Viral Pneumonia-1069.png,True,False,299,299,L,134.1,64a8d8c1d1291eda560195e416558272
Viral Pneumonia-107.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-107.png,Viral Pneumonia/images/Viral Pneumonia-107.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-107.png,Viral Pneumonia/masks/Viral Pneumonia-107.png,True,False,299,299,L,126.0,bf0f6c745c5a0b7e76d5a5ef95a61c1e
Viral Pneumonia-1070.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1070.png,Viral Pneumonia/images/Viral Pneumonia-1070.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1070.png,Viral Pneumonia/masks/Viral Pneumonia-1070.png,True,False,299,299,L,107.15,d9db867c9cded06b23050f128c38e838
Viral Pneumonia-1071.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1071.png,Viral Pneumonia/images/Viral Pneumonia-1071.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1071.png,Viral Pneumonia/masks/Viral Pneumonia-1071.png,True,False,299,299,L,141.83,6ae4051beb822f6ba8a6e796aaed57fb
Viral Pneumonia-1072.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1072.png,Viral Pneumonia/images/Viral Pneumonia-1072.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1072.png,Viral Pneumonia/masks/Viral Pneumonia-1072.png,True,False,299,299,L,145.16,32614d94a9a99cfed53be970d4ebd154
Viral Pneumonia-1073.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1073.png,Viral Pneumonia/images/Viral Pneumonia-1073.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1073.png,Viral Pneumonia/masks/Viral Pneumonia-1073.png,True,False,299,299,L,122.03,cf1e2a9687e3293bb0107076a7ea17e3
Viral Pneumonia-1074.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1074.png,Viral Pneumonia/images/Viral Pneumonia-1074.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1074.png,Viral Pneumonia/masks/Viral Pneumonia-1074.png,True,False,299,299,L,162.13,d236a896f8dac3ed8d8dee355c6f4ec8
Viral Pneumonia-1075.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1075.png,Viral Pneumonia/images/Viral Pneumonia-1075.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1075.png,Viral Pneumonia/masks/Viral Pneumonia-1075.png,True,False,299,299,L,155.11,5da07cda910b7ec1aff396264a681f64
Viral Pneumonia-1076.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1076.png,Viral Pneumonia/images/Viral Pneumonia-1076.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1076.png,Viral Pneumonia/masks/Viral Pneumonia-1076.png,True,False,299,299,L,124.89,440d8bb5b9b52054166062bba7b400c8
Viral Pneumonia-1077.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1077.png,Viral Pneumonia/images/Viral Pneumonia-1077.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1077.png,Viral Pneumonia/masks/Viral Pneumonia-1077.png,True,False,299,299,L,131.22,77c005c8460410d228d6778dd228157d
Viral Pneumonia-1078.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1078.png,Viral Pneumonia/images/Viral Pneumonia-1078.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1078.png,Viral Pneumonia/masks/Viral Pneumonia-1078.png,True,False,299,299,L,141.86,2cbb88f1a8138fc0f41344f054858090
Viral Pneumonia-1079.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1079.png,Viral Pneumonia/images/Viral Pneumonia-1079.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1079.png,Viral Pneumonia/masks/Viral Pneumonia-1079.png,True,False,299,299,L,150.8,0586f8becc0b2f3e10a485dd1eb74be7
Viral Pneumonia-108.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-108.png,Viral Pneumonia/images/Viral Pneumonia-108.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-108.png,Viral Pneumonia/masks/Viral Pneumonia-108.png,True,False,299,299,L,139.63,90f1c5d6946da13df42d6efc48f41d6c
Viral Pneumonia-1080.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1080.png,Viral Pneumonia/images/Viral Pneumonia-1080.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1080.png,Viral Pneumonia/masks/Viral Pneumonia-1080.png,True,False,299,299,L,142.48,88a4f99c61b40be2289aa8f543aeb8df
Viral Pneumonia-1081.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1081.png,Viral Pneumonia/images/Viral Pneumonia-1081.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1081.png,Viral Pneumonia/masks/Viral Pneumonia-1081.png,True,False,299,299,L,155.68,c61acad7dd83e15b11a59c4e203254f3
Viral Pneumonia-1082.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1082.png,Viral Pneumonia/images/Viral Pneumonia-1082.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1082.png,Viral Pneumonia/masks/Viral Pneumonia-1082.png,True,False,299,299,L,107.09,b3dca06468ebb621ace72b15e7e6c7ec
Viral Pneumonia-1083.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1083.png,Viral Pneumonia/images/Viral Pneumonia-1083.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1083.png,Viral Pneumonia/masks/Viral Pneumonia-1083.png,True,False,299,299,L,122.77,e13e54b14ce6a0185738935d39e6bee3
Viral Pneumonia-1084.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1084.png,Viral Pneumonia/images/Viral Pneumonia-1084.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1084.png,Viral Pneumonia/masks/Viral Pneumonia-1084.png,True,False,299,299,RGB,171.24,da164d732a26fb82672a9264558cf720
Viral Pneumonia-1085.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1085.png,Viral Pneumonia/images/Viral Pneumonia-1085.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1085.png,Viral Pneumonia/masks/Viral Pneumonia-1085.png,True,False,299,299,L,132.67,a8539bca0dd8cfede5917577c1b453fd
Viral Pneumonia-1086.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1086.png,Viral Pneumonia/images/Viral Pneumonia-1086.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1086.png,Viral Pneumonia/masks/Viral Pneumonia-1086.png,True,False,299,299,L,108.93,ed0370ea08579ab5cc8a4fc86e5e49db
Viral Pneumonia-1087.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1087.png,Viral Pneumonia/images/Viral Pneumonia-1087.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1087.png,Viral Pneumonia/masks/Viral Pneumonia-1087.png,True,False,299,299,L,143.52,963f0fb30de0b7dbf7d7ea014187061d
Viral Pneumonia-1088.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1088.png,Viral Pneumonia/images/Viral Pneumonia-1088.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1088.png,Viral Pneumonia/masks/Viral Pneumonia-1088.png,True,False,299,299,L,136.87,dd2bad79f32dccd309be123e09f257a4
Viral Pneumonia-1089.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1089.png,Viral Pneumonia/images/Viral Pneumonia-1089.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1089.png,Viral Pneumonia/masks/Viral Pneumonia-1089.png,True,False,299,299,L,135.61,fba997d51b7095e616b2f6c63d855a96
Viral Pneumonia-109.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-109.png,Viral Pneumonia/images/Viral Pneumonia-109.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-109.png,Viral Pneumonia/masks/Viral Pneumonia-109.png,True,False,299,299,L,143.07,6c8b056bf31d8aaaeb49b08fc0118e9d
Viral Pneumonia-1090.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1090.png,Viral Pneumonia/images/Viral Pneumonia-1090.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1090.png,Viral Pneumonia/masks/Viral Pneumonia-1090.png,True,False,299,299,L,88.6,85b0c0982c1199a0892d220534f5ab46
Viral Pneumonia-1091.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1091.png,Viral Pneumonia/images/Viral Pneumonia-1091.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1091.png,Viral Pneumonia/masks/Viral Pneumonia-1091.png,True,False,299,299,L,123.39,4c12d40582af36cd7667dd52ce369428
Viral Pneumonia-1092.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1092.png,Viral Pneumonia/images/Viral Pneumonia-1092.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1092.png,Viral Pneumonia/masks/Viral Pneumonia-1092.png,True,False,299,299,L,112.54,63d0de931193b37c27f5d38ddf7f9b13
Viral Pneumonia-1093.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1093.png,Viral Pneumonia/images/Viral Pneumonia-1093.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1093.png,Viral Pneumonia/masks/Viral Pneumonia-1093.png,True,False,299,299,L,113.47,4d5841ebf5e69f8a3d4bff9b8a471647
Viral Pneumonia-1094.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1094.png,Viral Pneumonia/images/Viral Pneumonia-1094.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1094.png,Viral Pneumonia/masks/Viral Pneumonia-1094.png,True,False,299,299,L,137.41,1e5016b23c7983011f4f88cd1af44b3f
Viral Pneumonia-1095.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1095.png,Viral Pneumonia/images/Viral Pneumonia-1095.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1095.png,Viral Pneumonia/masks/Viral Pneumonia-1095.png,True,False,299,299,L,122.39,0bbfcb98156db156941992f5a4ea8505
Viral Pneumonia-1096.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1096.png,Viral Pneumonia/images/Viral Pneumonia-1096.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1096.png,Viral Pneumonia/masks/Viral Pneumonia-1096.png,True,False,299,299,L,136.74,bc972281188ce459b68d0f7cf11951ef
Viral Pneumonia-1097.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1097.png,Viral Pneumonia/images/Viral Pneumonia-1097.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1097.png,Viral Pneumonia/masks/Viral Pneumonia-1097.png,True,False,299,299,L,132.61,b17fb524e841a55436aed133fd2d3123
Viral Pneumonia-1098.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1098.png,Viral Pneumonia/images/Viral Pneumonia-1098.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1098.png,Viral Pneumonia/masks/Viral Pneumonia-1098.png,True,False,299,299,L,138.77,3d20478331709f5d2e98793de3859c10
Viral Pneumonia-1099.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1099.png,Viral Pneumonia/images/Viral Pneumonia-1099.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1099.png,Viral Pneumonia/masks/Viral Pneumonia-1099.png,True,False,299,299,L,153.5,c072689f7155a310a4a52389314ab195
Viral Pneumonia-11.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-11.png,Viral Pneumonia/images/Viral Pneumonia-11.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-11.png,Viral Pneumonia/masks/Viral Pneumonia-11.png,True,False,299,299,L,104.91,f02e22c0b439ee59f125e6c885452ce1
Viral Pneumonia-110.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-110.png,Viral Pneumonia/images/Viral Pneumonia-110.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-110.png,Viral Pneumonia/masks/Viral Pneumonia-110.png,True,False,299,299,L,134.21,dba38b6a551aa5758df75950d429209b
Viral Pneumonia-1100.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1100.png,Viral Pneumonia/images/Viral Pneumonia-1100.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1100.png,Viral Pneumonia/masks/Viral Pneumonia-1100.png,True,False,299,299,L,128.49,3994e662cedc0d1e0e2d1098c26e2673
Viral Pneumonia-1101.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1101.png,Viral Pneumonia/images/Viral Pneumonia-1101.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1101.png,Viral Pneumonia/masks/Viral Pneumonia-1101.png,True,False,299,299,L,152.85,b3bb696b13b4434bd010c9048499d238
Viral Pneumonia-1102.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1102.png,Viral Pneumonia/images/Viral Pneumonia-1102.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1102.png,Viral Pneumonia/masks/Viral Pneumonia-1102.png,True,False,299,299,L,124.75,bb4f3abe6cced06d4e37e3a9de4a81b4
Viral Pneumonia-1103.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1103.png,Viral Pneumonia/images/Viral Pneumonia-1103.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1103.png,Viral Pneumonia/masks/Viral Pneumonia-1103.png,True,False,299,299,L,133.49,3dced5beef2d667cd16f3d564363b31e
Viral Pneumonia-1104.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1104.png,Viral Pneumonia/images/Viral Pneumonia-1104.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1104.png,Viral Pneumonia/masks/Viral Pneumonia-1104.png,True,False,299,299,L,128.38,b7092ccd08e334dfdb68d03b29d24299
Viral Pneumonia-1105.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1105.png,Viral Pneumonia/images/Viral Pneumonia-1105.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1105.png,Viral Pneumonia/masks/Viral Pneumonia-1105.png,True,False,299,299,L,120.25,0feeb9954bead5e2a5c8a8564505b3cd
Viral Pneumonia-1106.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1106.png,Viral Pneumonia/images/Viral Pneumonia-1106.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1106.png,Viral Pneumonia/masks/Viral Pneumonia-1106.png,True,False,299,299,L,119.93,6cabee390e758e53a43faed2c6ff6a0b
Viral Pneumonia-1107.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1107.png,Viral Pneumonia/images/Viral Pneumonia-1107.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1107.png,Viral Pneumonia/masks/Viral Pneumonia-1107.png,True,False,299,299,L,117.16,c0acbc9d54ccda37e8a447764722c0d0
Viral Pneumonia-1108.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1108.png,Viral Pneumonia/images/Viral Pneumonia-1108.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1108.png,Viral Pneumonia/masks/Viral Pneumonia-1108.png,True,False,299,299,L,124.05,60e2222921fe47a5aca60a997d7de974
Viral Pneumonia-1109.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1109.png,Viral Pneumonia/images/Viral Pneumonia-1109.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1109.png,Viral Pneumonia/masks/Viral Pneumonia-1109.png,True,False,299,299,L,115.08,032e2635dda11d6ec2511372bdc84e8d
Viral Pneumonia-111.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-111.png,Viral Pneumonia/images/Viral Pneumonia-111.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-111.png,Viral Pneumonia/masks/Viral Pneumonia-111.png,True,False,299,299,L,146.35,ad6679f9aa78307a6ccd0da18c0912da
Viral Pneumonia-1110.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1110.png,Viral Pneumonia/images/Viral Pneumonia-1110.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1110.png,Viral Pneumonia/masks/Viral Pneumonia-1110.png,True,False,299,299,L,138.2,2c6077d54f2144cd88feb5945430f449
Viral Pneumonia-1111.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1111.png,Viral Pneumonia/images/Viral Pneumonia-1111.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1111.png,Viral Pneumonia/masks/Viral Pneumonia-1111.png,True,False,299,299,L,126.89,6b19a4a1b5e9f2577a7f5843384936dc
Viral Pneumonia-1112.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1112.png,Viral Pneumonia/images/Viral Pneumonia-1112.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1112.png,Viral Pneumonia/masks/Viral Pneumonia-1112.png,True,False,299,299,L,116.95,3b24ec14e571bbfc69594c7ddf462928
Viral Pneumonia-1113.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1113.png,Viral Pneumonia/images/Viral Pneumonia-1113.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1113.png,Viral Pneumonia/masks/Viral Pneumonia-1113.png,True,False,299,299,L,138.78,e05c3e1150ab30919350dadd56f7c55b
Viral Pneumonia-1114.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1114.png,Viral Pneumonia/images/Viral Pneumonia-1114.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1114.png,Viral Pneumonia/masks/Viral Pneumonia-1114.png,True,False,299,299,L,125.4,430827b8c048f41a5283b19a6eccf3c9
Viral Pneumonia-1115.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1115.png,Viral Pneumonia/images/Viral Pneumonia-1115.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1115.png,Viral Pneumonia/masks/Viral Pneumonia-1115.png,True,False,299,299,L,129.08,5b7cc175df314d0baab8ac5e07abbf7f
Viral Pneumonia-1116.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1116.png,Viral Pneumonia/images/Viral Pneumonia-1116.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1116.png,Viral Pneumonia/masks/Viral Pneumonia-1116.png,True,False,299,299,L,87.54,aabf6e770d4eccdd079aafc84251b03e
Viral Pneumonia-1117.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1117.png,Viral Pneumonia/images/Viral Pneumonia-1117.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1117.png,Viral Pneumonia/masks/Viral Pneumonia-1117.png,True,False,299,299,L,141.08,86c1e56306d5fdeced78060bebb2f7cf
Viral Pneumonia-1118.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1118.png,Viral Pneumonia/images/Viral Pneumonia-1118.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1118.png,Viral Pneumonia/masks/Viral Pneumonia-1118.png,True,False,299,299,L,160.97,e21260922d49c5f10302cab9bd72aabb
Viral Pneumonia-1119.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1119.png,Viral Pneumonia/images/Viral Pneumonia-1119.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1119.png,Viral Pneumonia/masks/Viral Pneumonia-1119.png,True,False,299,299,L,147.26,720b02e22263c8998217953a34083155
Viral Pneumonia-112.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-112.png,Viral Pneumonia/images/Viral Pneumonia-112.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-112.png,Viral Pneumonia/masks/Viral Pneumonia-112.png,True,False,299,299,L,123.97,a97073bbc2646bbe10e5983b1db4a36e
Viral Pneumonia-1120.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1120.png,Viral Pneumonia/images/Viral Pneumonia-1120.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1120.png,Viral Pneumonia/masks/Viral Pneumonia-1120.png,True,False,299,299,L,140.2,c27113f373a810f46cb2ffcd6e22b3ba
Viral Pneumonia-1121.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1121.png,Viral Pneumonia/images/Viral Pneumonia-1121.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1121.png,Viral Pneumonia/masks/Viral Pneumonia-1121.png,True,False,299,299,L,105.31,f9c32b320614399b2dd3bea417dcb278
Viral Pneumonia-1122.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1122.png,Viral Pneumonia/images/Viral Pneumonia-1122.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1122.png,Viral Pneumonia/masks/Viral Pneumonia-1122.png,True,False,299,299,L,112.16,2086b8a81845843091b7dbeb88b2c937
Viral Pneumonia-1123.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1123.png,Viral Pneumonia/images/Viral Pneumonia-1123.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1123.png,Viral Pneumonia/masks/Viral Pneumonia-1123.png,True,False,299,299,RGB,145.07,84d5022193ba275fa4746f4e1e3c22d6
Viral Pneumonia-1124.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1124.png,Viral Pneumonia/images/Viral Pneumonia-1124.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1124.png,Viral Pneumonia/masks/Viral Pneumonia-1124.png,True,False,299,299,L,101.16,12f8c1258168c41f210d3b0f6439e769
Viral Pneumonia-1125.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1125.png,Viral Pneumonia/images/Viral Pneumonia-1125.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1125.png,Viral Pneumonia/masks/Viral Pneumonia-1125.png,True,False,299,299,L,120.97,af53c350a5016a9ec1d59edf00d88e09
Viral Pneumonia-1126.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1126.png,Viral Pneumonia/images/Viral Pneumonia-1126.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1126.png,Viral Pneumonia/masks/Viral Pneumonia-1126.png,True,False,299,299,L,139.22,480e044760887ed081255eca7de9f914
Viral Pneumonia-1127.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1127.png,Viral Pneumonia/images/Viral Pneumonia-1127.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1127.png,Viral Pneumonia/masks/Viral Pneumonia-1127.png,True,False,299,299,L,130.58,2a59dac26a0b363eb0f7f77bdadadaa0
Viral Pneumonia-1128.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1128.png,Viral Pneumonia/images/Viral Pneumonia-1128.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1128.png,Viral Pneumonia/masks/Viral Pneumonia-1128.png,True,False,299,299,L,105.76,e282125daca74faa677267d8fc21222c
Viral Pneumonia-1129.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1129.png,Viral Pneumonia/images/Viral Pneumonia-1129.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1129.png,Viral Pneumonia/masks/Viral Pneumonia-1129.png,True,False,299,299,L,121.76,74128dc10b22773901f640f2ebb87757
Viral Pneumonia-113.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-113.png,Viral Pneumonia/images/Viral Pneumonia-113.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-113.png,Viral Pneumonia/masks/Viral Pneumonia-113.png,True,False,299,299,L,138.04,9ce2bf85f75a197fd09d040a846907f2
Viral Pneumonia-1130.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1130.png,Viral Pneumonia/images/Viral Pneumonia-1130.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1130.png,Viral Pneumonia/masks/Viral Pneumonia-1130.png,True,False,299,299,L,150.07,6c4b9c0575de340097ffed9c1a1ac1d2
Viral Pneumonia-1131.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1131.png,Viral Pneumonia/images/Viral Pneumonia-1131.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1131.png,Viral Pneumonia/masks/Viral Pneumonia-1131.png,True,False,299,299,L,114.05,735e842e61480d9792c93ae9758c4a25
Viral Pneumonia-1132.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1132.png,Viral Pneumonia/images/Viral Pneumonia-1132.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1132.png,Viral Pneumonia/masks/Viral Pneumonia-1132.png,True,False,299,299,L,124.11,6f4a8b842d1ae17e7b98512eefaba069
Viral Pneumonia-1133.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1133.png,Viral Pneumonia/images/Viral Pneumonia-1133.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1133.png,Viral Pneumonia/masks/Viral Pneumonia-1133.png,True,False,299,299,L,151.71,2f5df19940f5550eb6461793e1edb055
Viral Pneumonia-1134.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1134.png,Viral Pneumonia/images/Viral Pneumonia-1134.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1134.png,Viral Pneumonia/masks/Viral Pneumonia-1134.png,True,False,299,299,L,143.56,cc051104118baf57080fee3bf00b007a
Viral Pneumonia-1135.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1135.png,Viral Pneumonia/images/Viral Pneumonia-1135.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1135.png,Viral Pneumonia/masks/Viral Pneumonia-1135.png,True,False,299,299,L,125.76,206c3289c013d14afeb2d58720f7067e
Viral Pneumonia-1136.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1136.png,Viral Pneumonia/images/Viral Pneumonia-1136.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1136.png,Viral Pneumonia/masks/Viral Pneumonia-1136.png,True,False,299,299,L,133.33,13fe7ee7060ad976dbac47bf00b0c7ea
Viral Pneumonia-1137.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1137.png,Viral Pneumonia/images/Viral Pneumonia-1137.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1137.png,Viral Pneumonia/masks/Viral Pneumonia-1137.png,True,False,299,299,L,157.4,ef4d3b16e161d0b22b08adb1fdd0ecc8
Viral Pneumonia-1138.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1138.png,Viral Pneumonia/images/Viral Pneumonia-1138.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1138.png,Viral Pneumonia/masks/Viral Pneumonia-1138.png,True,False,299,299,L,153.82,4542416720dc2e3272305854f1e1445a
Viral Pneumonia-1139.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1139.png,Viral Pneumonia/images/Viral Pneumonia-1139.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1139.png,Viral Pneumonia/masks/Viral Pneumonia-1139.png,True,False,299,299,L,161.45,ab335124ba23e05371fe6f652859ed67
Viral Pneumonia-114.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-114.png,Viral Pneumonia/images/Viral Pneumonia-114.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-114.png,Viral Pneumonia/masks/Viral Pneumonia-114.png,True,False,299,299,L,145.16,dfea645948eeea6354fe7befd07aeb4b
Viral Pneumonia-1140.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1140.png,Viral Pneumonia/images/Viral Pneumonia-1140.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1140.png,Viral Pneumonia/masks/Viral Pneumonia-1140.png,True,False,299,299,L,142.51,bcae88cf6e36ea5c5bab07e0e9f06b83
Viral Pneumonia-1141.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1141.png,Viral Pneumonia/images/Viral Pneumonia-1141.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1141.png,Viral Pneumonia/masks/Viral Pneumonia-1141.png,True,False,299,299,L,145.11,3e3ed0faed0180efee64319027640b4c
Viral Pneumonia-1142.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1142.png,Viral Pneumonia/images/Viral Pneumonia-1142.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1142.png,Viral Pneumonia/masks/Viral Pneumonia-1142.png,True,False,299,299,L,139.76,9b324b89168b15c93f2c2610e77cd301
Viral Pneumonia-1143.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1143.png,Viral Pneumonia/images/Viral Pneumonia-1143.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1143.png,Viral Pneumonia/masks/Viral Pneumonia-1143.png,True,False,299,299,L,138.56,f25bd41d8dbcbab3bf7fa8456d430d7e
Viral Pneumonia-1144.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1144.png,Viral Pneumonia/images/Viral Pneumonia-1144.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1144.png,Viral Pneumonia/masks/Viral Pneumonia-1144.png,True,False,299,299,L,121.14,52db57a5bc38ce8b1ee5506648d0d49d
Viral Pneumonia-1145.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1145.png,Viral Pneumonia/images/Viral Pneumonia-1145.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1145.png,Viral Pneumonia/masks/Viral Pneumonia-1145.png,True,False,299,299,L,150.03,719bf68a1afd75e1a959d43985c7ee21
Viral Pneumonia-1146.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1146.png,Viral Pneumonia/images/Viral Pneumonia-1146.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1146.png,Viral Pneumonia/masks/Viral Pneumonia-1146.png,True,False,299,299,L,156.38,d879e6ed33609972891f5c730d09ac81
Viral Pneumonia-1147.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1147.png,Viral Pneumonia/images/Viral Pneumonia-1147.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1147.png,Viral Pneumonia/masks/Viral Pneumonia-1147.png,True,False,299,299,L,147.1,1eb8925069e6034b079559ea93eb9b55
Viral Pneumonia-1148.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1148.png,Viral Pneumonia/images/Viral Pneumonia-1148.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1148.png,Viral Pneumonia/masks/Viral Pneumonia-1148.png,True,False,299,299,L,150.23,87be28cb3d1e8c5897afef828989f646
Viral Pneumonia-1149.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1149.png,Viral Pneumonia/images/Viral Pneumonia-1149.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1149.png,Viral Pneumonia/masks/Viral Pneumonia-1149.png,True,False,299,299,L,106.46,3f8fc1f35388a11e58ecf2bb7fc1df30
Viral Pneumonia-115.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-115.png,Viral Pneumonia/images/Viral Pneumonia-115.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-115.png,Viral Pneumonia/masks/Viral Pneumonia-115.png,True,False,299,299,L,134.16,1ef1908a02416a281f068ab3b0fcab25
Viral Pneumonia-1150.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1150.png,Viral Pneumonia/images/Viral Pneumonia-1150.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1150.png,Viral Pneumonia/masks/Viral Pneumonia-1150.png,True,False,299,299,L,117.65,f8964e10ab833f914f4c67d0b5baceb3
Viral Pneumonia-1151.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1151.png,Viral Pneumonia/images/Viral Pneumonia-1151.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1151.png,Viral Pneumonia/masks/Viral Pneumonia-1151.png,True,False,299,299,L,164.82,1c1257932ed38c920d3c5a8bb785cd07
Viral Pneumonia-1152.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1152.png,Viral Pneumonia/images/Viral Pneumonia-1152.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1152.png,Viral Pneumonia/masks/Viral Pneumonia-1152.png,True,False,299,299,L,166.94,dbb5b7b256e55a86491ae9eefab8750a
Viral Pneumonia-1153.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1153.png,Viral Pneumonia/images/Viral Pneumonia-1153.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1153.png,Viral Pneumonia/masks/Viral Pneumonia-1153.png,True,False,299,299,L,138.16,56dcd93afc8afaf82ba32a641a66d37e
Viral Pneumonia-1154.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1154.png,Viral Pneumonia/images/Viral Pneumonia-1154.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1154.png,Viral Pneumonia/masks/Viral Pneumonia-1154.png,True,False,299,299,RGB,136.19,9883f77b06d756587339540cd28c3014
Viral Pneumonia-1155.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1155.png,Viral Pneumonia/images/Viral Pneumonia-1155.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1155.png,Viral Pneumonia/masks/Viral Pneumonia-1155.png,True,False,299,299,RGB,137.12,3d80495e0e5595c03a81ddf7853635b0
Viral Pneumonia-1156.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1156.png,Viral Pneumonia/images/Viral Pneumonia-1156.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1156.png,Viral Pneumonia/masks/Viral Pneumonia-1156.png,True,False,299,299,L,150.0,1cba22ebf1bcf3455cccb9ee5a7d1a7f
Viral Pneumonia-1157.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1157.png,Viral Pneumonia/images/Viral Pneumonia-1157.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1157.png,Viral Pneumonia/masks/Viral Pneumonia-1157.png,True,False,299,299,L,109.76,dc0b899b01650a0007706e1160281139
Viral Pneumonia-1158.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1158.png,Viral Pneumonia/images/Viral Pneumonia-1158.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1158.png,Viral Pneumonia/masks/Viral Pneumonia-1158.png,True,False,299,299,L,95.94,622f46fa1e6a43a8f0807fb45ee26609
Viral Pneumonia-1159.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1159.png,Viral Pneumonia/images/Viral Pneumonia-1159.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1159.png,Viral Pneumonia/masks/Viral Pneumonia-1159.png,True,False,299,299,L,145.18,3531d9a6983985d99a6d63837343831d
Viral Pneumonia-116.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-116.png,Viral Pneumonia/images/Viral Pneumonia-116.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-116.png,Viral Pneumonia/masks/Viral Pneumonia-116.png,True,False,299,299,L,144.44,bf977323ed29f57decce47e417a3ded8
Viral Pneumonia-1160.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1160.png,Viral Pneumonia/images/Viral Pneumonia-1160.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1160.png,Viral Pneumonia/masks/Viral Pneumonia-1160.png,True,False,299,299,L,123.84,682974d205bec300eaf8acc4425dc289
Viral Pneumonia-1161.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1161.png,Viral Pneumonia/images/Viral Pneumonia-1161.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1161.png,Viral Pneumonia/masks/Viral Pneumonia-1161.png,True,False,299,299,L,85.13,e5b39a02a3dc482e316d8472f8141783
Viral Pneumonia-1162.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1162.png,Viral Pneumonia/images/Viral Pneumonia-1162.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1162.png,Viral Pneumonia/masks/Viral Pneumonia-1162.png,True,False,299,299,L,133.4,0d9eeda47ef24e94fd53378e378a081b
Viral Pneumonia-1163.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1163.png,Viral Pneumonia/images/Viral Pneumonia-1163.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1163.png,Viral Pneumonia/masks/Viral Pneumonia-1163.png,True,False,299,299,L,129.87,48bf69dbeea9703e8fc6eb516afd4eac
Viral Pneumonia-1164.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1164.png,Viral Pneumonia/images/Viral Pneumonia-1164.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1164.png,Viral Pneumonia/masks/Viral Pneumonia-1164.png,True,False,299,299,L,141.98,cd64f59b8b0fadc508ff0ce084ea35ca
Viral Pneumonia-1165.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1165.png,Viral Pneumonia/images/Viral Pneumonia-1165.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1165.png,Viral Pneumonia/masks/Viral Pneumonia-1165.png,True,False,299,299,L,119.65,ccd3a1776fbce135cbdfd088bfca49dc
Viral Pneumonia-1166.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1166.png,Viral Pneumonia/images/Viral Pneumonia-1166.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1166.png,Viral Pneumonia/masks/Viral Pneumonia-1166.png,True,False,299,299,L,118.63,9db932bb509294e901557b2b4e4bc110
Viral Pneumonia-1167.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1167.png,Viral Pneumonia/images/Viral Pneumonia-1167.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1167.png,Viral Pneumonia/masks/Viral Pneumonia-1167.png,True,False,299,299,L,78.88,46c9012aba30d93dba6c4dd9a425d32b
Viral Pneumonia-1168.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1168.png,Viral Pneumonia/images/Viral Pneumonia-1168.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1168.png,Viral Pneumonia/masks/Viral Pneumonia-1168.png,True,False,299,299,L,104.88,81c731932607c6be48228929e5911d32
Viral Pneumonia-1169.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1169.png,Viral Pneumonia/images/Viral Pneumonia-1169.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1169.png,Viral Pneumonia/masks/Viral Pneumonia-1169.png,True,False,299,299,L,122.99,42c06b53deb2ad878b368740ee3ac046
Viral Pneumonia-117.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-117.png,Viral Pneumonia/images/Viral Pneumonia-117.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-117.png,Viral Pneumonia/masks/Viral Pneumonia-117.png,True,False,299,299,L,131.94,84ffdd23e87c71c2da6a75b61cd82d6f
Viral Pneumonia-1170.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1170.png,Viral Pneumonia/images/Viral Pneumonia-1170.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1170.png,Viral Pneumonia/masks/Viral Pneumonia-1170.png,True,False,299,299,L,137.61,b17368121f31411c66f5ba0fe35ebb16
Viral Pneumonia-1171.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1171.png,Viral Pneumonia/images/Viral Pneumonia-1171.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1171.png,Viral Pneumonia/masks/Viral Pneumonia-1171.png,True,False,299,299,L,83.03,9c7ac8e8f7249d1253f1608d7f56374d
Viral Pneumonia-1172.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1172.png,Viral Pneumonia/images/Viral Pneumonia-1172.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1172.png,Viral Pneumonia/masks/Viral Pneumonia-1172.png,True,False,299,299,L,93.06,df2e74cecf1238397981d8c5e2d6b637
Viral Pneumonia-1173.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1173.png,Viral Pneumonia/images/Viral Pneumonia-1173.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1173.png,Viral Pneumonia/masks/Viral Pneumonia-1173.png,True,False,299,299,L,123.0,64d67aa6f04b7226ea5dc5ce41d078c5
Viral Pneumonia-1174.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1174.png,Viral Pneumonia/images/Viral Pneumonia-1174.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1174.png,Viral Pneumonia/masks/Viral Pneumonia-1174.png,True,False,299,299,L,114.9,da7ce770d56161cf0a8af2bbd0d642a8
Viral Pneumonia-1175.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1175.png,Viral Pneumonia/images/Viral Pneumonia-1175.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1175.png,Viral Pneumonia/masks/Viral Pneumonia-1175.png,True,False,299,299,L,122.2,9d607e83bc05b3234a3f08c90e088205
Viral Pneumonia-1176.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1176.png,Viral Pneumonia/images/Viral Pneumonia-1176.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1176.png,Viral Pneumonia/masks/Viral Pneumonia-1176.png,True,False,299,299,L,108.53,98323c0fd73fceea32a10565a49fd266
Viral Pneumonia-1177.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1177.png,Viral Pneumonia/images/Viral Pneumonia-1177.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1177.png,Viral Pneumonia/masks/Viral Pneumonia-1177.png,True,False,299,299,L,115.35,a426f150586899af36cdd10ecb9a2042
Viral Pneumonia-1178.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1178.png,Viral Pneumonia/images/Viral Pneumonia-1178.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1178.png,Viral Pneumonia/masks/Viral Pneumonia-1178.png,True,False,299,299,L,127.02,4ece98a4d1081e523a9ef97a0a9ba78b
Viral Pneumonia-1179.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1179.png,Viral Pneumonia/images/Viral Pneumonia-1179.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1179.png,Viral Pneumonia/masks/Viral Pneumonia-1179.png,True,False,299,299,L,156.35,ebbed078f0f9bd96314653fdfe5e2dbf
Viral Pneumonia-118.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-118.png,Viral Pneumonia/images/Viral Pneumonia-118.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-118.png,Viral Pneumonia/masks/Viral Pneumonia-118.png,True,False,299,299,L,131.94,84ffdd23e87c71c2da6a75b61cd82d6f
Viral Pneumonia-1180.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1180.png,Viral Pneumonia/images/Viral Pneumonia-1180.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1180.png,Viral Pneumonia/masks/Viral Pneumonia-1180.png,True,False,299,299,L,135.84,198378ac79e61ed28030457ad334aeb3
Viral Pneumonia-1181.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1181.png,Viral Pneumonia/images/Viral Pneumonia-1181.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1181.png,Viral Pneumonia/masks/Viral Pneumonia-1181.png,True,False,299,299,L,127.82,60f07738268a3d002c9dc86773c1c885
Viral Pneumonia-1182.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1182.png,Viral Pneumonia/images/Viral Pneumonia-1182.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1182.png,Viral Pneumonia/masks/Viral Pneumonia-1182.png,True,False,299,299,L,77.66,894306ec38bde4da279591aea90c5398
Viral Pneumonia-1183.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1183.png,Viral Pneumonia/images/Viral Pneumonia-1183.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1183.png,Viral Pneumonia/masks/Viral Pneumonia-1183.png,True,False,299,299,L,135.49,684216cf66a1e2e7752c83bcb0d8e11e
Viral Pneumonia-1184.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1184.png,Viral Pneumonia/images/Viral Pneumonia-1184.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1184.png,Viral Pneumonia/masks/Viral Pneumonia-1184.png,True,False,299,299,L,133.68,9e3c9d97625d806df450bab73d140643
Viral Pneumonia-1185.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1185.png,Viral Pneumonia/images/Viral Pneumonia-1185.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1185.png,Viral Pneumonia/masks/Viral Pneumonia-1185.png,True,False,299,299,L,126.63,09fa1709019bd582800a0c00a9dcb2fd
Viral Pneumonia-1186.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1186.png,Viral Pneumonia/images/Viral Pneumonia-1186.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1186.png,Viral Pneumonia/masks/Viral Pneumonia-1186.png,True,False,299,299,L,121.61,0927eca3c98faad0a60ef334d79d0fcf
Viral Pneumonia-1187.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1187.png,Viral Pneumonia/images/Viral Pneumonia-1187.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1187.png,Viral Pneumonia/masks/Viral Pneumonia-1187.png,True,False,299,299,L,156.46,bfb2c70ebda831e30a461d547ff0bdce
Viral Pneumonia-1188.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1188.png,Viral Pneumonia/images/Viral Pneumonia-1188.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1188.png,Viral Pneumonia/masks/Viral Pneumonia-1188.png,True,False,299,299,L,124.54,d5fb4e907e97845c0ab01c2a878d508d
Viral Pneumonia-1189.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1189.png,Viral Pneumonia/images/Viral Pneumonia-1189.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1189.png,Viral Pneumonia/masks/Viral Pneumonia-1189.png,True,False,299,299,L,108.53,13b4c3448a4c27a8f90ce2807eb8b4f2
Viral Pneumonia-119.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-119.png,Viral Pneumonia/images/Viral Pneumonia-119.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-119.png,Viral Pneumonia/masks/Viral Pneumonia-119.png,True,False,299,299,L,83.85,eb65d6f32c27f05e981fa9a878c41ca5
Viral Pneumonia-1190.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1190.png,Viral Pneumonia/images/Viral Pneumonia-1190.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1190.png,Viral Pneumonia/masks/Viral Pneumonia-1190.png,True,False,299,299,L,127.0,281ab6006aea3445ac892fb0c0eb6453
Viral Pneumonia-1191.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1191.png,Viral Pneumonia/images/Viral Pneumonia-1191.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1191.png,Viral Pneumonia/masks/Viral Pneumonia-1191.png,True,False,299,299,L,109.86,c762d4cf69adc994751c39e4904c341b
Viral Pneumonia-1192.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1192.png,Viral Pneumonia/images/Viral Pneumonia-1192.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1192.png,Viral Pneumonia/masks/Viral Pneumonia-1192.png,True,False,299,299,L,92.92,06d74e7dc739111eb8db513375e41934
Viral Pneumonia-1193.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1193.png,Viral Pneumonia/images/Viral Pneumonia-1193.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1193.png,Viral Pneumonia/masks/Viral Pneumonia-1193.png,True,False,299,299,L,142.85,41dcb844ba4418b5b67d3bbeefb44cbb
Viral Pneumonia-1194.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1194.png,Viral Pneumonia/images/Viral Pneumonia-1194.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1194.png,Viral Pneumonia/masks/Viral Pneumonia-1194.png,True,False,299,299,L,122.45,fd9bb61a9df2f0b9874521b4b238afda
Viral Pneumonia-1195.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1195.png,Viral Pneumonia/images/Viral Pneumonia-1195.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1195.png,Viral Pneumonia/masks/Viral Pneumonia-1195.png,True,False,299,299,L,158.96,caa631b80421a88516f2362d9376813c
Viral Pneumonia-1196.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1196.png,Viral Pneumonia/images/Viral Pneumonia-1196.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1196.png,Viral Pneumonia/masks/Viral Pneumonia-1196.png,True,False,299,299,L,145.43,fbc58e2aa021b06a8697410d7420b0af
Viral Pneumonia-1197.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1197.png,Viral Pneumonia/images/Viral Pneumonia-1197.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1197.png,Viral Pneumonia/masks/Viral Pneumonia-1197.png,True,False,299,299,L,83.89,90fcf36057c9d6e119638be644ae69c7
Viral Pneumonia-1198.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1198.png,Viral Pneumonia/images/Viral Pneumonia-1198.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1198.png,Viral Pneumonia/masks/Viral Pneumonia-1198.png,True,False,299,299,L,111.9,e76fe1e5e3df94a876d10396b2a68a12
Viral Pneumonia-1199.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1199.png,Viral Pneumonia/images/Viral Pneumonia-1199.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1199.png,Viral Pneumonia/masks/Viral Pneumonia-1199.png,True,False,299,299,L,132.39,f0957fd958ab6df4a663a68fb3839d24
Viral Pneumonia-12.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-12.png,Viral Pneumonia/images/Viral Pneumonia-12.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-12.png,Viral Pneumonia/masks/Viral Pneumonia-12.png,True,False,299,299,L,64.77,0c4a0ff9ccc0e121e73b96ef6f9d0c74
Viral Pneumonia-120.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-120.png,Viral Pneumonia/images/Viral Pneumonia-120.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-120.png,Viral Pneumonia/masks/Viral Pneumonia-120.png,True,False,299,299,L,132.88,eb5301050632045f9ab8a6fe1676a280
Viral Pneumonia-1200.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1200.png,Viral Pneumonia/images/Viral Pneumonia-1200.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1200.png,Viral Pneumonia/masks/Viral Pneumonia-1200.png,True,False,299,299,L,155.1,e221014e9db257c3549c26c6406caa36
Viral Pneumonia-1201.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1201.png,Viral Pneumonia/images/Viral Pneumonia-1201.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1201.png,Viral Pneumonia/masks/Viral Pneumonia-1201.png,True,False,299,299,L,167.11,f72b5b99569f71d1e17646b7ab3826d3
Viral Pneumonia-1202.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1202.png,Viral Pneumonia/images/Viral Pneumonia-1202.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1202.png,Viral Pneumonia/masks/Viral Pneumonia-1202.png,True,False,299,299,L,110.29,97a9589cbf3f67cdd615bd23a2fc8bf8
Viral Pneumonia-1203.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1203.png,Viral Pneumonia/images/Viral Pneumonia-1203.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1203.png,Viral Pneumonia/masks/Viral Pneumonia-1203.png,True,False,299,299,L,125.29,ba5c611fcb757f96299f4c7f71de8018
Viral Pneumonia-1204.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1204.png,Viral Pneumonia/images/Viral Pneumonia-1204.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1204.png,Viral Pneumonia/masks/Viral Pneumonia-1204.png,True,False,299,299,L,101.27,06ce72d4ef75e26683d25a054875cc94
Viral Pneumonia-1205.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1205.png,Viral Pneumonia/images/Viral Pneumonia-1205.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1205.png,Viral Pneumonia/masks/Viral Pneumonia-1205.png,True,False,299,299,L,118.5,08d3ce571a98c0647160f680402bce44
Viral Pneumonia-1206.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1206.png,Viral Pneumonia/images/Viral Pneumonia-1206.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1206.png,Viral Pneumonia/masks/Viral Pneumonia-1206.png,True,False,299,299,L,90.7,5119d23c7e4f65a4da31342c4f7e49f8
Viral Pneumonia-1207.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1207.png,Viral Pneumonia/images/Viral Pneumonia-1207.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1207.png,Viral Pneumonia/masks/Viral Pneumonia-1207.png,True,False,299,299,L,120.93,59c2413411bb860696c303f17b75ba84
Viral Pneumonia-1208.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1208.png,Viral Pneumonia/images/Viral Pneumonia-1208.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1208.png,Viral Pneumonia/masks/Viral Pneumonia-1208.png,True,False,299,299,L,107.62,582ed48796c1e10d24cdc49de441ac5f
Viral Pneumonia-1209.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1209.png,Viral Pneumonia/images/Viral Pneumonia-1209.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1209.png,Viral Pneumonia/masks/Viral Pneumonia-1209.png,True,False,299,299,L,122.92,689409551aec23a0110b236568dfd534
Viral Pneumonia-121.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-121.png,Viral Pneumonia/images/Viral Pneumonia-121.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-121.png,Viral Pneumonia/masks/Viral Pneumonia-121.png,True,False,299,299,L,140.38,bafdc147dd6acc47aba1ac85d04621f5
Viral Pneumonia-1210.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1210.png,Viral Pneumonia/images/Viral Pneumonia-1210.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1210.png,Viral Pneumonia/masks/Viral Pneumonia-1210.png,True,False,299,299,L,147.98,744f3bb34f2f41a779d5ca32f09f4c17
Viral Pneumonia-1211.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1211.png,Viral Pneumonia/images/Viral Pneumonia-1211.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1211.png,Viral Pneumonia/masks/Viral Pneumonia-1211.png,True,False,299,299,L,121.94,ce3e0c9e184260b6ed2ee4df5af7035b
Viral Pneumonia-1212.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1212.png,Viral Pneumonia/images/Viral Pneumonia-1212.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1212.png,Viral Pneumonia/masks/Viral Pneumonia-1212.png,True,False,299,299,L,132.42,38d773f301f55581533fb705c7f70cde
Viral Pneumonia-1213.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1213.png,Viral Pneumonia/images/Viral Pneumonia-1213.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1213.png,Viral Pneumonia/masks/Viral Pneumonia-1213.png,True,False,299,299,L,107.94,92bc5ee96e3f5f652702ef16d338b5cd
Viral Pneumonia-1214.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1214.png,Viral Pneumonia/images/Viral Pneumonia-1214.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1214.png,Viral Pneumonia/masks/Viral Pneumonia-1214.png,True,False,299,299,L,99.12,ccb33bba6a828b00a665e48a5dbd97e7
Viral Pneumonia-1215.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1215.png,Viral Pneumonia/images/Viral Pneumonia-1215.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1215.png,Viral Pneumonia/masks/Viral Pneumonia-1215.png,True,False,299,299,L,104.52,07cbf013066e06671f37f45f28058e59
Viral Pneumonia-1216.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1216.png,Viral Pneumonia/images/Viral Pneumonia-1216.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1216.png,Viral Pneumonia/masks/Viral Pneumonia-1216.png,True,False,299,299,L,108.41,492258412fe91c5d64839cea68e61758
Viral Pneumonia-1217.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1217.png,Viral Pneumonia/images/Viral Pneumonia-1217.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1217.png,Viral Pneumonia/masks/Viral Pneumonia-1217.png,True,False,299,299,L,102.94,c66a13eaa0a4b23efb72681117674de6
Viral Pneumonia-1218.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1218.png,Viral Pneumonia/images/Viral Pneumonia-1218.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1218.png,Viral Pneumonia/masks/Viral Pneumonia-1218.png,True,False,299,299,L,116.12,daeeed139d6c308c9cfd713dc2aecca6
Viral Pneumonia-1219.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1219.png,Viral Pneumonia/images/Viral Pneumonia-1219.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1219.png,Viral Pneumonia/masks/Viral Pneumonia-1219.png,True,False,299,299,L,125.41,5a0c766714509c8ed6c799e26b19f966
Viral Pneumonia-122.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-122.png,Viral Pneumonia/images/Viral Pneumonia-122.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-122.png,Viral Pneumonia/masks/Viral Pneumonia-122.png,True,False,299,299,L,127.79,b9f2b375daee0f1b5e4746f943f1a95a
Viral Pneumonia-1220.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1220.png,Viral Pneumonia/images/Viral Pneumonia-1220.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1220.png,Viral Pneumonia/masks/Viral Pneumonia-1220.png,True,False,299,299,L,114.07,0ecdef89aa80b20e69281ab41a34bad5
Viral Pneumonia-1221.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1221.png,Viral Pneumonia/images/Viral Pneumonia-1221.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1221.png,Viral Pneumonia/masks/Viral Pneumonia-1221.png,True,False,299,299,L,111.26,bd030018047f5c991507c0d6d15e0a7c
Viral Pneumonia-1222.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1222.png,Viral Pneumonia/images/Viral Pneumonia-1222.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1222.png,Viral Pneumonia/masks/Viral Pneumonia-1222.png,True,False,299,299,L,132.75,5d0ecb4f20aa8ccae57f1735fcb41b2d
Viral Pneumonia-1223.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1223.png,Viral Pneumonia/images/Viral Pneumonia-1223.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1223.png,Viral Pneumonia/masks/Viral Pneumonia-1223.png,True,False,299,299,L,108.1,9cb0ffef7dbf7e54d9cc0fe25b97db56
Viral Pneumonia-1224.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1224.png,Viral Pneumonia/images/Viral Pneumonia-1224.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1224.png,Viral Pneumonia/masks/Viral Pneumonia-1224.png,True,False,299,299,L,98.96,eb2fbb7052ac74d07bbc9fddc0ecf6e2
Viral Pneumonia-1225.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1225.png,Viral Pneumonia/images/Viral Pneumonia-1225.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1225.png,Viral Pneumonia/masks/Viral Pneumonia-1225.png,True,False,299,299,L,98.07,773920733c9bffef93e59263daed1008
Viral Pneumonia-1226.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1226.png,Viral Pneumonia/images/Viral Pneumonia-1226.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1226.png,Viral Pneumonia/masks/Viral Pneumonia-1226.png,True,False,299,299,L,128.48,434c6a7dea36c537adf7aabb27420be3
Viral Pneumonia-1227.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1227.png,Viral Pneumonia/images/Viral Pneumonia-1227.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1227.png,Viral Pneumonia/masks/Viral Pneumonia-1227.png,True,False,299,299,L,111.53,17c3dddaa6142e6371ad5f454d112990
Viral Pneumonia-1228.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1228.png,Viral Pneumonia/images/Viral Pneumonia-1228.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1228.png,Viral Pneumonia/masks/Viral Pneumonia-1228.png,True,False,299,299,L,114.77,065f119af02af44980aad4b952b659cd
Viral Pneumonia-1229.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1229.png,Viral Pneumonia/images/Viral Pneumonia-1229.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1229.png,Viral Pneumonia/masks/Viral Pneumonia-1229.png,True,False,299,299,L,120.18,636d69cf83e5f19ebd6a8f23e64ee20b
Viral Pneumonia-123.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-123.png,Viral Pneumonia/images/Viral Pneumonia-123.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-123.png,Viral Pneumonia/masks/Viral Pneumonia-123.png,True,False,299,299,L,134.1,e3fe06fc00e32b93a0fb086a0f27dff8
Viral Pneumonia-1230.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1230.png,Viral Pneumonia/images/Viral Pneumonia-1230.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1230.png,Viral Pneumonia/masks/Viral Pneumonia-1230.png,True,False,299,299,L,114.42,bf029683832c6f2386192fdc102e07c7
Viral Pneumonia-1231.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1231.png,Viral Pneumonia/images/Viral Pneumonia-1231.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1231.png,Viral Pneumonia/masks/Viral Pneumonia-1231.png,True,False,299,299,L,127.55,1b63f97b26eebe723c5f9ac6ac60f8d5
Viral Pneumonia-1232.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1232.png,Viral Pneumonia/images/Viral Pneumonia-1232.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1232.png,Viral Pneumonia/masks/Viral Pneumonia-1232.png,True,False,299,299,L,107.08,2ac14ee6281d9bf08e5d3f8e8bae7016
Viral Pneumonia-1233.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1233.png,Viral Pneumonia/images/Viral Pneumonia-1233.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1233.png,Viral Pneumonia/masks/Viral Pneumonia-1233.png,True,False,299,299,L,130.08,280f9f797a2e9f1a3272caf10269221a
Viral Pneumonia-1234.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1234.png,Viral Pneumonia/images/Viral Pneumonia-1234.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1234.png,Viral Pneumonia/masks/Viral Pneumonia-1234.png,True,False,299,299,L,119.95,16d77ad0b1c15d8f3c2dae0827ae1577
Viral Pneumonia-1235.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1235.png,Viral Pneumonia/images/Viral Pneumonia-1235.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1235.png,Viral Pneumonia/masks/Viral Pneumonia-1235.png,True,False,299,299,L,97.85,c9801a56db5f6f1e3b4c1db231003150
Viral Pneumonia-1236.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1236.png,Viral Pneumonia/images/Viral Pneumonia-1236.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1236.png,Viral Pneumonia/masks/Viral Pneumonia-1236.png,True,False,299,299,L,128.66,a7421fb346d56afcd279f5082aa96239
Viral Pneumonia-1237.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1237.png,Viral Pneumonia/images/Viral Pneumonia-1237.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1237.png,Viral Pneumonia/masks/Viral Pneumonia-1237.png,True,False,299,299,L,108.65,09940f4e0847ce779d123518f1ab71e1
Viral Pneumonia-1238.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1238.png,Viral Pneumonia/images/Viral Pneumonia-1238.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1238.png,Viral Pneumonia/masks/Viral Pneumonia-1238.png,True,False,299,299,L,152.2,79f9ff7a9bc07d9a87fb412b472ab24b
Viral Pneumonia-1239.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1239.png,Viral Pneumonia/images/Viral Pneumonia-1239.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1239.png,Viral Pneumonia/masks/Viral Pneumonia-1239.png,True,False,299,299,L,87.44,ac2156c82692c1766260729c7ca5f62c
Viral Pneumonia-124.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-124.png,Viral Pneumonia/images/Viral Pneumonia-124.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-124.png,Viral Pneumonia/masks/Viral Pneumonia-124.png,True,False,299,299,L,116.61,10c672f8d87c64ea6d72944f1604e0b9
Viral Pneumonia-1240.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1240.png,Viral Pneumonia/images/Viral Pneumonia-1240.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1240.png,Viral Pneumonia/masks/Viral Pneumonia-1240.png,True,False,299,299,L,83.17,a109d74dbc4c433afc7d23a049356fcf
Viral Pneumonia-1241.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1241.png,Viral Pneumonia/images/Viral Pneumonia-1241.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1241.png,Viral Pneumonia/masks/Viral Pneumonia-1241.png,True,False,299,299,L,127.3,f7d04be7b6c607e637b5ccc21a17b689
Viral Pneumonia-1242.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1242.png,Viral Pneumonia/images/Viral Pneumonia-1242.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1242.png,Viral Pneumonia/masks/Viral Pneumonia-1242.png,True,False,299,299,L,102.34,a0a475103be2fcfe7208d3a0bad161c2
Viral Pneumonia-1243.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1243.png,Viral Pneumonia/images/Viral Pneumonia-1243.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1243.png,Viral Pneumonia/masks/Viral Pneumonia-1243.png,True,False,299,299,L,140.52,1b2b26b550b5c00846e37a0c9cfe28ef
Viral Pneumonia-1244.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1244.png,Viral Pneumonia/images/Viral Pneumonia-1244.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1244.png,Viral Pneumonia/masks/Viral Pneumonia-1244.png,True,False,299,299,L,125.82,93d29accbd18caa59d7f7b4248101c64
Viral Pneumonia-1245.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1245.png,Viral Pneumonia/images/Viral Pneumonia-1245.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1245.png,Viral Pneumonia/masks/Viral Pneumonia-1245.png,True,False,299,299,L,99.96,aa69068e61074b2ec43c8167722b2b01
Viral Pneumonia-1246.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1246.png,Viral Pneumonia/images/Viral Pneumonia-1246.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1246.png,Viral Pneumonia/masks/Viral Pneumonia-1246.png,True,False,299,299,L,116.7,990ebddf615cf854f43f9773b4f87f56
Viral Pneumonia-1247.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1247.png,Viral Pneumonia/images/Viral Pneumonia-1247.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1247.png,Viral Pneumonia/masks/Viral Pneumonia-1247.png,True,False,299,299,L,126.05,7375b2ea3a1b820ea600f3877c6d9b61
Viral Pneumonia-1248.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1248.png,Viral Pneumonia/images/Viral Pneumonia-1248.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1248.png,Viral Pneumonia/masks/Viral Pneumonia-1248.png,True,False,299,299,L,125.15,839177931fcdf7fb919074f97e307713
Viral Pneumonia-1249.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1249.png,Viral Pneumonia/images/Viral Pneumonia-1249.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1249.png,Viral Pneumonia/masks/Viral Pneumonia-1249.png,True,False,299,299,L,117.25,750d78e6efe384a2e3997a15ab274a42
Viral Pneumonia-125.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-125.png,Viral Pneumonia/images/Viral Pneumonia-125.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-125.png,Viral Pneumonia/masks/Viral Pneumonia-125.png,True,False,299,299,L,131.0,46d8935690db37a0e4cd22af83142189
Viral Pneumonia-1250.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1250.png,Viral Pneumonia/images/Viral Pneumonia-1250.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1250.png,Viral Pneumonia/masks/Viral Pneumonia-1250.png,True,False,299,299,L,106.32,30b0f58c5b660eede1bb4bdbe7dc27d4
Viral Pneumonia-1251.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1251.png,Viral Pneumonia/images/Viral Pneumonia-1251.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1251.png,Viral Pneumonia/masks/Viral Pneumonia-1251.png,True,False,299,299,L,115.98,a3036749b1a0729222f21d4eab04f10e
Viral Pneumonia-1252.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1252.png,Viral Pneumonia/images/Viral Pneumonia-1252.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1252.png,Viral Pneumonia/masks/Viral Pneumonia-1252.png,True,False,299,299,L,126.96,880fc06ff4c1efe146e43932a8de1327
Viral Pneumonia-1253.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1253.png,Viral Pneumonia/images/Viral Pneumonia-1253.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1253.png,Viral Pneumonia/masks/Viral Pneumonia-1253.png,True,False,299,299,L,111.74,182ee17e33a6f6270e644a689dc60883
Viral Pneumonia-1254.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1254.png,Viral Pneumonia/images/Viral Pneumonia-1254.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1254.png,Viral Pneumonia/masks/Viral Pneumonia-1254.png,True,False,299,299,L,118.02,38981b4a9c1db609ae57b17052b57231
Viral Pneumonia-1255.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1255.png,Viral Pneumonia/images/Viral Pneumonia-1255.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1255.png,Viral Pneumonia/masks/Viral Pneumonia-1255.png,True,False,299,299,L,121.01,cd426ea6d4cc1223f798d59e793adcac
Viral Pneumonia-1256.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1256.png,Viral Pneumonia/images/Viral Pneumonia-1256.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1256.png,Viral Pneumonia/masks/Viral Pneumonia-1256.png,True,False,299,299,L,112.91,7feb19467a6439bef377c03574f0c004
Viral Pneumonia-1257.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1257.png,Viral Pneumonia/images/Viral Pneumonia-1257.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1257.png,Viral Pneumonia/masks/Viral Pneumonia-1257.png,True,False,299,299,L,119.64,f7ae5a96c2f9ad427451154096a30732
Viral Pneumonia-1258.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1258.png,Viral Pneumonia/images/Viral Pneumonia-1258.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1258.png,Viral Pneumonia/masks/Viral Pneumonia-1258.png,True,False,299,299,L,114.49,6f950434bedfff69e7fbf7ddad0c7846
Viral Pneumonia-1259.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1259.png,Viral Pneumonia/images/Viral Pneumonia-1259.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1259.png,Viral Pneumonia/masks/Viral Pneumonia-1259.png,True,False,299,299,L,76.2,5672f0cb841c4ab8eb89c681435febe9
Viral Pneumonia-126.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-126.png,Viral Pneumonia/images/Viral Pneumonia-126.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-126.png,Viral Pneumonia/masks/Viral Pneumonia-126.png,True,False,299,299,L,129.33,fe6b0b13f447e665a964cab1d29ad1f7
Viral Pneumonia-1260.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1260.png,Viral Pneumonia/images/Viral Pneumonia-1260.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1260.png,Viral Pneumonia/masks/Viral Pneumonia-1260.png,True,False,299,299,L,137.06,165bfd0a8038d0886e9ba8230244fdc9
Viral Pneumonia-1261.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1261.png,Viral Pneumonia/images/Viral Pneumonia-1261.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1261.png,Viral Pneumonia/masks/Viral Pneumonia-1261.png,True,False,299,299,L,137.57,e717136c8e02ae187cd5d8d4dcd040a2
Viral Pneumonia-1262.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1262.png,Viral Pneumonia/images/Viral Pneumonia-1262.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1262.png,Viral Pneumonia/masks/Viral Pneumonia-1262.png,True,False,299,299,L,130.66,44c884d8fe7f04d19e218a40627c2cfd
Viral Pneumonia-1263.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1263.png,Viral Pneumonia/images/Viral Pneumonia-1263.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1263.png,Viral Pneumonia/masks/Viral Pneumonia-1263.png,True,False,299,299,L,115.67,d661c947ee1e332047a4e4aebfdd2629
Viral Pneumonia-1264.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1264.png,Viral Pneumonia/images/Viral Pneumonia-1264.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1264.png,Viral Pneumonia/masks/Viral Pneumonia-1264.png,True,False,299,299,L,120.7,de43df051e5dc889d88f8a73d8499502
Viral Pneumonia-1265.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1265.png,Viral Pneumonia/images/Viral Pneumonia-1265.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1265.png,Viral Pneumonia/masks/Viral Pneumonia-1265.png,True,False,299,299,L,102.21,31ef5dd9fe98ce157da88efd71e168ff
Viral Pneumonia-1266.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1266.png,Viral Pneumonia/images/Viral Pneumonia-1266.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1266.png,Viral Pneumonia/masks/Viral Pneumonia-1266.png,True,False,299,299,L,156.86,0349ca81fa6132862bc75f16f92dfabb
Viral Pneumonia-1267.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1267.png,Viral Pneumonia/images/Viral Pneumonia-1267.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1267.png,Viral Pneumonia/masks/Viral Pneumonia-1267.png,True,False,299,299,L,140.01,f9ac4c260606f2b997a655abce2336f6
Viral Pneumonia-1268.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1268.png,Viral Pneumonia/images/Viral Pneumonia-1268.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1268.png,Viral Pneumonia/masks/Viral Pneumonia-1268.png,True,False,299,299,L,147.35,a27c269b18c125f54f3a2838e3a9bf0f
Viral Pneumonia-1269.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1269.png,Viral Pneumonia/images/Viral Pneumonia-1269.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1269.png,Viral Pneumonia/masks/Viral Pneumonia-1269.png,True,False,299,299,L,112.29,a0d01a894f3e9ebae2bf2e8217b53dc0
Viral Pneumonia-127.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-127.png,Viral Pneumonia/images/Viral Pneumonia-127.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-127.png,Viral Pneumonia/masks/Viral Pneumonia-127.png,True,False,299,299,L,120.54,687772530b3a5c7af917d0266f2393d2
Viral Pneumonia-1270.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1270.png,Viral Pneumonia/images/Viral Pneumonia-1270.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1270.png,Viral Pneumonia/masks/Viral Pneumonia-1270.png,True,False,299,299,L,135.83,0f174bcfe06e276b7001fa9450a42ce5
Viral Pneumonia-1271.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1271.png,Viral Pneumonia/images/Viral Pneumonia-1271.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1271.png,Viral Pneumonia/masks/Viral Pneumonia-1271.png,True,False,299,299,L,110.95,f503ecd08eac2c98f0d07c9ca29b1662
Viral Pneumonia-1272.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1272.png,Viral Pneumonia/images/Viral Pneumonia-1272.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1272.png,Viral Pneumonia/masks/Viral Pneumonia-1272.png,True,False,299,299,L,148.76,c8d726237946565a07263a399cfc979f
Viral Pneumonia-1273.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1273.png,Viral Pneumonia/images/Viral Pneumonia-1273.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1273.png,Viral Pneumonia/masks/Viral Pneumonia-1273.png,True,False,299,299,L,150.65,bdc5b0313747b530b4b54882d38c48ac
Viral Pneumonia-1274.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1274.png,Viral Pneumonia/images/Viral Pneumonia-1274.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1274.png,Viral Pneumonia/masks/Viral Pneumonia-1274.png,True,False,299,299,L,120.73,c944c65ee92aa6010d89aa07bd34f4d9
Viral Pneumonia-1275.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1275.png,Viral Pneumonia/images/Viral Pneumonia-1275.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1275.png,Viral Pneumonia/masks/Viral Pneumonia-1275.png,True,False,299,299,L,138.7,ff0ee243d4aec07bde4a9f0e3e66c259
Viral Pneumonia-1276.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1276.png,Viral Pneumonia/images/Viral Pneumonia-1276.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1276.png,Viral Pneumonia/masks/Viral Pneumonia-1276.png,True,False,299,299,L,138.43,9112a7150cc594af144402e96e292f14
Viral Pneumonia-1277.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1277.png,Viral Pneumonia/images/Viral Pneumonia-1277.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1277.png,Viral Pneumonia/masks/Viral Pneumonia-1277.png,True,False,299,299,L,134.86,027a786e1c208a1b30e6872892259aab
Viral Pneumonia-1278.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1278.png,Viral Pneumonia/images/Viral Pneumonia-1278.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1278.png,Viral Pneumonia/masks/Viral Pneumonia-1278.png,True,False,299,299,L,151.03,74a5d902b9ab6e8a2c92e640253b861e
Viral Pneumonia-1279.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1279.png,Viral Pneumonia/images/Viral Pneumonia-1279.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1279.png,Viral Pneumonia/masks/Viral Pneumonia-1279.png,True,False,299,299,L,137.78,91088c96afbccd60b3ec6566d4fe6d18
Viral Pneumonia-128.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-128.png,Viral Pneumonia/images/Viral Pneumonia-128.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-128.png,Viral Pneumonia/masks/Viral Pneumonia-128.png,True,False,299,299,L,139.77,e67a76370458f4fd72035f07a3ac9c55
Viral Pneumonia-1280.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1280.png,Viral Pneumonia/images/Viral Pneumonia-1280.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1280.png,Viral Pneumonia/masks/Viral Pneumonia-1280.png,True,False,299,299,L,128.23,d97b86f6c2b376f47f379d2f0a17547e
Viral Pneumonia-1281.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1281.png,Viral Pneumonia/images/Viral Pneumonia-1281.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1281.png,Viral Pneumonia/masks/Viral Pneumonia-1281.png,True,False,299,299,L,193.69,5b5c310623ded0680e4ddcd2b5a7f32e
Viral Pneumonia-1282.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1282.png,Viral Pneumonia/images/Viral Pneumonia-1282.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1282.png,Viral Pneumonia/masks/Viral Pneumonia-1282.png,True,False,299,299,L,128.0,7b62dda831134d9b2b4bdd5bdc5d3b25
Viral Pneumonia-1283.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1283.png,Viral Pneumonia/images/Viral Pneumonia-1283.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1283.png,Viral Pneumonia/masks/Viral Pneumonia-1283.png,True,False,299,299,L,128.48,186b265c88320975f866c46a47fc4f85
Viral Pneumonia-1284.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1284.png,Viral Pneumonia/images/Viral Pneumonia-1284.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1284.png,Viral Pneumonia/masks/Viral Pneumonia-1284.png,True,False,299,299,L,148.21,ca095bd2d03ac4c75988aea4f06f287b
Viral Pneumonia-1285.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1285.png,Viral Pneumonia/images/Viral Pneumonia-1285.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1285.png,Viral Pneumonia/masks/Viral Pneumonia-1285.png,True,False,299,299,L,134.94,7e2e7761dcd2e05423efb7ad5e391614
Viral Pneumonia-1286.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1286.png,Viral Pneumonia/images/Viral Pneumonia-1286.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1286.png,Viral Pneumonia/masks/Viral Pneumonia-1286.png,True,False,299,299,L,130.67,f6afdc0d7353d364ad4ef090c927a724
Viral Pneumonia-1287.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1287.png,Viral Pneumonia/images/Viral Pneumonia-1287.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1287.png,Viral Pneumonia/masks/Viral Pneumonia-1287.png,True,False,299,299,L,143.63,6e5bbd6709b42d8d4aca9ffa34f57e52
Viral Pneumonia-1288.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1288.png,Viral Pneumonia/images/Viral Pneumonia-1288.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1288.png,Viral Pneumonia/masks/Viral Pneumonia-1288.png,True,False,299,299,L,137.3,c0574cc9b084b4c9d894893ec0f4a14e
Viral Pneumonia-1289.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1289.png,Viral Pneumonia/images/Viral Pneumonia-1289.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1289.png,Viral Pneumonia/masks/Viral Pneumonia-1289.png,True,False,299,299,L,127.76,688b29c347edd89003a8cfa86a1951f5
Viral Pneumonia-129.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-129.png,Viral Pneumonia/images/Viral Pneumonia-129.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-129.png,Viral Pneumonia/masks/Viral Pneumonia-129.png,True,False,299,299,L,112.44,2dd83787008badaf5ae8bf71f21eaf73
Viral Pneumonia-1290.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1290.png,Viral Pneumonia/images/Viral Pneumonia-1290.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1290.png,Viral Pneumonia/masks/Viral Pneumonia-1290.png,True,False,299,299,L,119.42,f99e1e61a3240fb56f3c7b60f7b4fefe
Viral Pneumonia-1291.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1291.png,Viral Pneumonia/images/Viral Pneumonia-1291.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1291.png,Viral Pneumonia/masks/Viral Pneumonia-1291.png,True,False,299,299,L,146.98,54cd23875b215082665a4709526fe1d3
Viral Pneumonia-1292.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1292.png,Viral Pneumonia/images/Viral Pneumonia-1292.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1292.png,Viral Pneumonia/masks/Viral Pneumonia-1292.png,True,False,299,299,L,112.19,38c549943c803d4ab89fadd0cd62797d
Viral Pneumonia-1293.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1293.png,Viral Pneumonia/images/Viral Pneumonia-1293.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1293.png,Viral Pneumonia/masks/Viral Pneumonia-1293.png,True,False,299,299,L,125.57,09b8dd36f050014ab7a317b04a953a07
Viral Pneumonia-1294.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1294.png,Viral Pneumonia/images/Viral Pneumonia-1294.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1294.png,Viral Pneumonia/masks/Viral Pneumonia-1294.png,True,False,299,299,L,134.32,b26443ce199e3d026728d411dbb1b309
Viral Pneumonia-1295.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1295.png,Viral Pneumonia/images/Viral Pneumonia-1295.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1295.png,Viral Pneumonia/masks/Viral Pneumonia-1295.png,True,False,299,299,L,112.34,d3df50e7db513bdc6da6991a102b322d
Viral Pneumonia-1296.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1296.png,Viral Pneumonia/images/Viral Pneumonia-1296.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1296.png,Viral Pneumonia/masks/Viral Pneumonia-1296.png,True,False,299,299,L,143.21,82796369750a3f8eba52bd3ca5d553ed
Viral Pneumonia-1297.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1297.png,Viral Pneumonia/images/Viral Pneumonia-1297.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1297.png,Viral Pneumonia/masks/Viral Pneumonia-1297.png,True,False,299,299,L,120.27,2615d038638708f9ccc8f07ef011de9b
Viral Pneumonia-1298.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1298.png,Viral Pneumonia/images/Viral Pneumonia-1298.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1298.png,Viral Pneumonia/masks/Viral Pneumonia-1298.png,True,False,299,299,L,142.95,cf1a2ccc8fdef96462bb48b448a19dd6
Viral Pneumonia-1299.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1299.png,Viral Pneumonia/images/Viral Pneumonia-1299.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1299.png,Viral Pneumonia/masks/Viral Pneumonia-1299.png,True,False,299,299,L,106.68,97aaa40d3902dafae81394209a467cad
Viral Pneumonia-13.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-13.png,Viral Pneumonia/images/Viral Pneumonia-13.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-13.png,Viral Pneumonia/masks/Viral Pneumonia-13.png,True,False,299,299,L,137.23,0a6cf73b05cddfb4cf718e5ef27accf3
Viral Pneumonia-130.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-130.png,Viral Pneumonia/images/Viral Pneumonia-130.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-130.png,Viral Pneumonia/masks/Viral Pneumonia-130.png,True,False,299,299,L,129.43,edde31a22450f56d60df820b1d6737c4
Viral Pneumonia-1300.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1300.png,Viral Pneumonia/images/Viral Pneumonia-1300.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1300.png,Viral Pneumonia/masks/Viral Pneumonia-1300.png,True,False,299,299,L,130.91,f74bec877725b1f33e6efcb279b1be33
Viral Pneumonia-1301.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1301.png,Viral Pneumonia/images/Viral Pneumonia-1301.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1301.png,Viral Pneumonia/masks/Viral Pneumonia-1301.png,True,False,299,299,L,134.72,07b811f5852422b1e661f04525c5ecae
Viral Pneumonia-1302.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1302.png,Viral Pneumonia/images/Viral Pneumonia-1302.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1302.png,Viral Pneumonia/masks/Viral Pneumonia-1302.png,True,False,299,299,L,133.73,9bd46877fd9e7a571bd47c635e21a99c
Viral Pneumonia-1303.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1303.png,Viral Pneumonia/images/Viral Pneumonia-1303.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1303.png,Viral Pneumonia/masks/Viral Pneumonia-1303.png,True,False,299,299,L,143.12,a26ff61c6e3c29285d569136c060932d
Viral Pneumonia-1304.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1304.png,Viral Pneumonia/images/Viral Pneumonia-1304.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1304.png,Viral Pneumonia/masks/Viral Pneumonia-1304.png,True,False,299,299,L,129.36,41159b3babbe4436e5984133f5249313
Viral Pneumonia-1305.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1305.png,Viral Pneumonia/images/Viral Pneumonia-1305.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1305.png,Viral Pneumonia/masks/Viral Pneumonia-1305.png,True,False,299,299,L,151.42,d61a6aa9bc16cd532195fd2d5e808201
Viral Pneumonia-1306.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1306.png,Viral Pneumonia/images/Viral Pneumonia-1306.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1306.png,Viral Pneumonia/masks/Viral Pneumonia-1306.png,True,False,299,299,L,126.51,8b41f43b7cda488dee15cb162fd0c7ff
Viral Pneumonia-1307.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1307.png,Viral Pneumonia/images/Viral Pneumonia-1307.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1307.png,Viral Pneumonia/masks/Viral Pneumonia-1307.png,True,False,299,299,L,116.8,43c2172726d28cf2feb0650005a4b686
Viral Pneumonia-1308.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1308.png,Viral Pneumonia/images/Viral Pneumonia-1308.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1308.png,Viral Pneumonia/masks/Viral Pneumonia-1308.png,True,False,299,299,L,171.45,ad3f63c5d05a72ff3ab63da2bf9a6863
Viral Pneumonia-1309.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1309.png,Viral Pneumonia/images/Viral Pneumonia-1309.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1309.png,Viral Pneumonia/masks/Viral Pneumonia-1309.png,True,False,299,299,L,131.1,ebc117f0ca583910561bcd49d838198f
Viral Pneumonia-131.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-131.png,Viral Pneumonia/images/Viral Pneumonia-131.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-131.png,Viral Pneumonia/masks/Viral Pneumonia-131.png,True,False,299,299,L,127.75,e77b3397ab90b9326385f921dbcd7b62
Viral Pneumonia-1310.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1310.png,Viral Pneumonia/images/Viral Pneumonia-1310.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1310.png,Viral Pneumonia/masks/Viral Pneumonia-1310.png,True,False,299,299,L,146.83,493c8bd7a9a9c47e58ce806456210cc6
Viral Pneumonia-1311.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1311.png,Viral Pneumonia/images/Viral Pneumonia-1311.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1311.png,Viral Pneumonia/masks/Viral Pneumonia-1311.png,True,False,299,299,L,113.28,75b2a9b97020abf70448cee22b51e5ae
Viral Pneumonia-1312.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1312.png,Viral Pneumonia/images/Viral Pneumonia-1312.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1312.png,Viral Pneumonia/masks/Viral Pneumonia-1312.png,True,False,299,299,L,120.66,0423b6403fa47bacb7a79b8426294859
Viral Pneumonia-1313.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1313.png,Viral Pneumonia/images/Viral Pneumonia-1313.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1313.png,Viral Pneumonia/masks/Viral Pneumonia-1313.png,True,False,299,299,L,125.77,16f250c3bedaca0d10a509734653c31c
Viral Pneumonia-1314.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1314.png,Viral Pneumonia/images/Viral Pneumonia-1314.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1314.png,Viral Pneumonia/masks/Viral Pneumonia-1314.png,True,False,299,299,L,111.7,7069644ac7cd2ed64bda6e3dc8e49339
Viral Pneumonia-1315.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1315.png,Viral Pneumonia/images/Viral Pneumonia-1315.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1315.png,Viral Pneumonia/masks/Viral Pneumonia-1315.png,True,False,299,299,L,116.34,e9acab1ca548b751da0c0dc69d93f2d0
Viral Pneumonia-1316.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1316.png,Viral Pneumonia/images/Viral Pneumonia-1316.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1316.png,Viral Pneumonia/masks/Viral Pneumonia-1316.png,True,False,299,299,L,180.87,b2e577b99496a513916dacecc2ef209a
Viral Pneumonia-1317.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1317.png,Viral Pneumonia/images/Viral Pneumonia-1317.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1317.png,Viral Pneumonia/masks/Viral Pneumonia-1317.png,True,False,299,299,L,127.0,686f056e115d764ac8a111657d527411
Viral Pneumonia-1318.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1318.png,Viral Pneumonia/images/Viral Pneumonia-1318.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1318.png,Viral Pneumonia/masks/Viral Pneumonia-1318.png,True,False,299,299,L,130.44,f5a52f1259c55672b2423454422ffb9f
Viral Pneumonia-1319.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1319.png,Viral Pneumonia/images/Viral Pneumonia-1319.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1319.png,Viral Pneumonia/masks/Viral Pneumonia-1319.png,True,False,299,299,L,126.21,5945d294e68bfcc950d2ff710678724e
Viral Pneumonia-132.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-132.png,Viral Pneumonia/images/Viral Pneumonia-132.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-132.png,Viral Pneumonia/masks/Viral Pneumonia-132.png,True,False,299,299,L,128.76,64c2e44bbacc29731cd3ac727de989b7
Viral Pneumonia-1320.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1320.png,Viral Pneumonia/images/Viral Pneumonia-1320.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1320.png,Viral Pneumonia/masks/Viral Pneumonia-1320.png,True,False,299,299,L,131.33,bd25d0e1938c5d8272173df237e7c44c
Viral Pneumonia-1321.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1321.png,Viral Pneumonia/images/Viral Pneumonia-1321.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1321.png,Viral Pneumonia/masks/Viral Pneumonia-1321.png,True,False,299,299,L,117.94,4abd6f52e75834ac830311cdb42d165b
Viral Pneumonia-1322.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1322.png,Viral Pneumonia/images/Viral Pneumonia-1322.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1322.png,Viral Pneumonia/masks/Viral Pneumonia-1322.png,True,False,299,299,L,145.44,e4418c49660c7eba83cb74eaa430d509
Viral Pneumonia-1323.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1323.png,Viral Pneumonia/images/Viral Pneumonia-1323.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1323.png,Viral Pneumonia/masks/Viral Pneumonia-1323.png,True,False,299,299,L,137.41,21e411981177dbae3a1013c6e405d87c
Viral Pneumonia-1324.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1324.png,Viral Pneumonia/images/Viral Pneumonia-1324.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1324.png,Viral Pneumonia/masks/Viral Pneumonia-1324.png,True,False,299,299,L,124.26,145d02dac14c51b420c125688a8ff8a2
Viral Pneumonia-1325.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1325.png,Viral Pneumonia/images/Viral Pneumonia-1325.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1325.png,Viral Pneumonia/masks/Viral Pneumonia-1325.png,True,False,299,299,L,109.5,9a20a71a66f7524e5f0cbe595eb537f0
Viral Pneumonia-1326.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1326.png,Viral Pneumonia/images/Viral Pneumonia-1326.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1326.png,Viral Pneumonia/masks/Viral Pneumonia-1326.png,True,False,299,299,L,135.3,1fa07c1d787f351c1a4f74b922842bad
Viral Pneumonia-1327.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1327.png,Viral Pneumonia/images/Viral Pneumonia-1327.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1327.png,Viral Pneumonia/masks/Viral Pneumonia-1327.png,True,False,299,299,L,168.25,7204799b214b1ac29b1709311fb202d2
Viral Pneumonia-1328.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1328.png,Viral Pneumonia/images/Viral Pneumonia-1328.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1328.png,Viral Pneumonia/masks/Viral Pneumonia-1328.png,True,False,299,299,L,119.41,259ceb561b9bdb8c651df29ca0a94066
Viral Pneumonia-1329.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1329.png,Viral Pneumonia/images/Viral Pneumonia-1329.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1329.png,Viral Pneumonia/masks/Viral Pneumonia-1329.png,True,False,299,299,L,117.42,503500ccdea513a35179c4d9da88632a
Viral Pneumonia-133.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-133.png,Viral Pneumonia/images/Viral Pneumonia-133.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-133.png,Viral Pneumonia/masks/Viral Pneumonia-133.png,True,False,299,299,L,129.63,fe14b32baab3a6bf076a027a10c7a829
Viral Pneumonia-1330.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1330.png,Viral Pneumonia/images/Viral Pneumonia-1330.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1330.png,Viral Pneumonia/masks/Viral Pneumonia-1330.png,True,False,299,299,L,160.58,7ad59cb4c3ad6cf74715ee137dc9b2be
Viral Pneumonia-1331.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1331.png,Viral Pneumonia/images/Viral Pneumonia-1331.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1331.png,Viral Pneumonia/masks/Viral Pneumonia-1331.png,True,False,299,299,L,136.43,84da65a0fcd7efa24d1d2860e28fa5fa
Viral Pneumonia-1332.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1332.png,Viral Pneumonia/images/Viral Pneumonia-1332.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1332.png,Viral Pneumonia/masks/Viral Pneumonia-1332.png,True,False,299,299,L,106.06,edfcb5d8876012d381382f0f6c3593d2
Viral Pneumonia-1333.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1333.png,Viral Pneumonia/images/Viral Pneumonia-1333.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1333.png,Viral Pneumonia/masks/Viral Pneumonia-1333.png,True,False,299,299,L,147.67,e5929a4623ddfedbd33954b644e537e5
Viral Pneumonia-1334.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1334.png,Viral Pneumonia/images/Viral Pneumonia-1334.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1334.png,Viral Pneumonia/masks/Viral Pneumonia-1334.png,True,False,299,299,L,134.7,d047637bea0e8ad68a6bd77c0f264c29
Viral Pneumonia-1335.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1335.png,Viral Pneumonia/images/Viral Pneumonia-1335.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1335.png,Viral Pneumonia/masks/Viral Pneumonia-1335.png,True,False,299,299,L,128.05,a1aae055967905835b06973d94cc8aa6
Viral Pneumonia-1336.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1336.png,Viral Pneumonia/images/Viral Pneumonia-1336.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1336.png,Viral Pneumonia/masks/Viral Pneumonia-1336.png,True,False,299,299,L,150.61,d44475d255e9f87f67a27af8cf66f89f
Viral Pneumonia-1337.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1337.png,Viral Pneumonia/images/Viral Pneumonia-1337.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1337.png,Viral Pneumonia/masks/Viral Pneumonia-1337.png,True,False,299,299,L,117.82,ea60e4b5db2131408446d4634a7e6fe1
Viral Pneumonia-1338.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1338.png,Viral Pneumonia/images/Viral Pneumonia-1338.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1338.png,Viral Pneumonia/masks/Viral Pneumonia-1338.png,True,False,299,299,L,129.06,d5a650aed16f15a5c9243869a66de6bf
Viral Pneumonia-1339.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1339.png,Viral Pneumonia/images/Viral Pneumonia-1339.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1339.png,Viral Pneumonia/masks/Viral Pneumonia-1339.png,True,False,299,299,L,133.23,ca1a679488a137a73df7e507f7c97b1c
Viral Pneumonia-134.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-134.png,Viral Pneumonia/images/Viral Pneumonia-134.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-134.png,Viral Pneumonia/masks/Viral Pneumonia-134.png,True,False,299,299,L,147.91,72d13186151e398877c34cf696216bdf
Viral Pneumonia-1340.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1340.png,Viral Pneumonia/images/Viral Pneumonia-1340.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1340.png,Viral Pneumonia/masks/Viral Pneumonia-1340.png,True,False,299,299,L,148.71,8461941c980ec5c451d66494caebf67d
Viral Pneumonia-1341.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1341.png,Viral Pneumonia/images/Viral Pneumonia-1341.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1341.png,Viral Pneumonia/masks/Viral Pneumonia-1341.png,True,False,299,299,L,143.14,1807f33d587f4e356fe31c755742c39d
Viral Pneumonia-1342.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1342.png,Viral Pneumonia/images/Viral Pneumonia-1342.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1342.png,Viral Pneumonia/masks/Viral Pneumonia-1342.png,True,False,299,299,L,129.8,d0f5c885438e702d67c14c141b77592b
Viral Pneumonia-1343.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1343.png,Viral Pneumonia/images/Viral Pneumonia-1343.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1343.png,Viral Pneumonia/masks/Viral Pneumonia-1343.png,True,False,299,299,L,89.58,ed25fc9d62436cb4cf4205feda5bc053
Viral Pneumonia-1344.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1344.png,Viral Pneumonia/images/Viral Pneumonia-1344.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1344.png,Viral Pneumonia/masks/Viral Pneumonia-1344.png,True,False,299,299,L,138.0,90f2fd9447ed2796be40a973b935dd32
Viral Pneumonia-1345.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-1345.png,Viral Pneumonia/images/Viral Pneumonia-1345.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-1345.png,Viral Pneumonia/masks/Viral Pneumonia-1345.png,True,False,299,299,L,131.06,dccc00ca5707610be8a300d71f7bbb03
Viral Pneumonia-135.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-135.png,Viral Pneumonia/images/Viral Pneumonia-135.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-135.png,Viral Pneumonia/masks/Viral Pneumonia-135.png,True,False,299,299,L,138.32,43a86ac20acdf3d265d6be3555c25d3b
Viral Pneumonia-136.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-136.png,Viral Pneumonia/images/Viral Pneumonia-136.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-136.png,Viral Pneumonia/masks/Viral Pneumonia-136.png,True,False,299,299,L,116.83,649c0b083c73916e1198e6982e0bc706
Viral Pneumonia-137.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-137.png,Viral Pneumonia/images/Viral Pneumonia-137.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-137.png,Viral Pneumonia/masks/Viral Pneumonia-137.png,True,False,299,299,L,96.91,67a082790ebd7a82869858bc44b3fe05
Viral Pneumonia-138.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-138.png,Viral Pneumonia/images/Viral Pneumonia-138.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-138.png,Viral Pneumonia/masks/Viral Pneumonia-138.png,True,False,299,299,L,115.96,4e8f08c38ca51be5bd40a928763aab24
Viral Pneumonia-139.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-139.png,Viral Pneumonia/images/Viral Pneumonia-139.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-139.png,Viral Pneumonia/masks/Viral Pneumonia-139.png,True,False,299,299,L,91.96,5894657ff5e6b1126f3cdd8ceadd4419
Viral Pneumonia-14.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-14.png,Viral Pneumonia/images/Viral Pneumonia-14.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-14.png,Viral Pneumonia/masks/Viral Pneumonia-14.png,True,False,299,299,L,112.46,da5d9e3227be8f946bcf05caaf03df7e
Viral Pneumonia-140.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-140.png,Viral Pneumonia/images/Viral Pneumonia-140.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-140.png,Viral Pneumonia/masks/Viral Pneumonia-140.png,True,False,299,299,RGB,130.64,82f999a0bdb0a6003351d5624b7a00bd
Viral Pneumonia-141.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-141.png,Viral Pneumonia/images/Viral Pneumonia-141.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-141.png,Viral Pneumonia/masks/Viral Pneumonia-141.png,True,False,299,299,L,118.46,dba4683ddd1c45d9d8e3ad3fb6e0ea30
Viral Pneumonia-142.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-142.png,Viral Pneumonia/images/Viral Pneumonia-142.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-142.png,Viral Pneumonia/masks/Viral Pneumonia-142.png,True,False,299,299,RGB,71.16,49470f1a7ce182280dfd884843e760dc
Viral Pneumonia-143.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-143.png,Viral Pneumonia/images/Viral Pneumonia-143.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-143.png,Viral Pneumonia/masks/Viral Pneumonia-143.png,True,False,299,299,L,130.35,0a9d258255ae703f048fccbb379698f0
Viral Pneumonia-144.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-144.png,Viral Pneumonia/images/Viral Pneumonia-144.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-144.png,Viral Pneumonia/masks/Viral Pneumonia-144.png,True,False,299,299,L,75.31,c810d9eadbdb7db5728a324654f1a14a
Viral Pneumonia-145.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-145.png,Viral Pneumonia/images/Viral Pneumonia-145.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-145.png,Viral Pneumonia/masks/Viral Pneumonia-145.png,True,False,299,299,L,137.43,15d4fc6d668568249465980e6d71c14f
Viral Pneumonia-146.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-146.png,Viral Pneumonia/images/Viral Pneumonia-146.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-146.png,Viral Pneumonia/masks/Viral Pneumonia-146.png,True,False,299,299,L,79.12,5ba8152e6622b52b1208a390f84bd78c
Viral Pneumonia-147.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-147.png,Viral Pneumonia/images/Viral Pneumonia-147.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-147.png,Viral Pneumonia/masks/Viral Pneumonia-147.png,True,False,299,299,L,140.98,ff35265a046bde1c3d2d45e6612238a0
Viral Pneumonia-148.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-148.png,Viral Pneumonia/images/Viral Pneumonia-148.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-148.png,Viral Pneumonia/masks/Viral Pneumonia-148.png,True,False,299,299,L,118.52,d2c8784d629a1c535bc2723d67f96431
Viral Pneumonia-149.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-149.png,Viral Pneumonia/images/Viral Pneumonia-149.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-149.png,Viral Pneumonia/masks/Viral Pneumonia-149.png,True,False,299,299,L,118.62,7380d233317103e1024b5f6c8268f013
Viral Pneumonia-15.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-15.png,Viral Pneumonia/images/Viral Pneumonia-15.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-15.png,Viral Pneumonia/masks/Viral Pneumonia-15.png,True,False,299,299,L,77.19,d1d1eb65e6f6e572c977a5e03ecaeab1
Viral Pneumonia-150.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-150.png,Viral Pneumonia/images/Viral Pneumonia-150.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-150.png,Viral Pneumonia/masks/Viral Pneumonia-150.png,True,False,299,299,RGB,122.1,093699278a28e5d5012970f492529b95
Viral Pneumonia-151.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-151.png,Viral Pneumonia/images/Viral Pneumonia-151.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-151.png,Viral Pneumonia/masks/Viral Pneumonia-151.png,True,False,299,299,RGB,164.85,b08da33b7c7d4a4fcc403b62c0054f25
Viral Pneumonia-152.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-152.png,Viral Pneumonia/images/Viral Pneumonia-152.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-152.png,Viral Pneumonia/masks/Viral Pneumonia-152.png,True,False,299,299,L,128.83,0106acd24ef33c445afae12a06aa359c
Viral Pneumonia-153.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-153.png,Viral Pneumonia/images/Viral Pneumonia-153.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-153.png,Viral Pneumonia/masks/Viral Pneumonia-153.png,True,False,299,299,L,127.93,f044201461945c402901cb4e5750e3eb
Viral Pneumonia-154.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-154.png,Viral Pneumonia/images/Viral Pneumonia-154.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-154.png,Viral Pneumonia/masks/Viral Pneumonia-154.png,True,False,299,299,L,105.08,64f72715579883da978ea36f0f41b31a
Viral Pneumonia-155.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-155.png,Viral Pneumonia/images/Viral Pneumonia-155.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-155.png,Viral Pneumonia/masks/Viral Pneumonia-155.png,True,False,299,299,L,158.57,f5a44e2f218beacf5ab79984251cd2cf
Viral Pneumonia-156.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-156.png,Viral Pneumonia/images/Viral Pneumonia-156.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-156.png,Viral Pneumonia/masks/Viral Pneumonia-156.png,True,False,299,299,L,100.6,d78ccc38bb5b834b7b0e3687c507d26a
Viral Pneumonia-157.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-157.png,Viral Pneumonia/images/Viral Pneumonia-157.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-157.png,Viral Pneumonia/masks/Viral Pneumonia-157.png,True,False,299,299,L,82.09,35ba352314dc1c122154d84e01a2fae9
Viral Pneumonia-158.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-158.png,Viral Pneumonia/images/Viral Pneumonia-158.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-158.png,Viral Pneumonia/masks/Viral Pneumonia-158.png,True,False,299,299,RGB,130.1,559ff07016ed1f984a95917b462110c1
Viral Pneumonia-159.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-159.png,Viral Pneumonia/images/Viral Pneumonia-159.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-159.png,Viral Pneumonia/masks/Viral Pneumonia-159.png,True,False,299,299,L,77.17,c3fe5b2ba56e8f321cd8e4d40ecb0a01
Viral Pneumonia-16.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-16.png,Viral Pneumonia/images/Viral Pneumonia-16.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-16.png,Viral Pneumonia/masks/Viral Pneumonia-16.png,True,False,299,299,L,123.8,41d217aaa79718b74b241aeef2ab2e69
Viral Pneumonia-160.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-160.png,Viral Pneumonia/images/Viral Pneumonia-160.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-160.png,Viral Pneumonia/masks/Viral Pneumonia-160.png,True,False,299,299,L,131.63,e3a2c842c22e2ed7e587d4fba7926d97
Viral Pneumonia-161.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-161.png,Viral Pneumonia/images/Viral Pneumonia-161.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-161.png,Viral Pneumonia/masks/Viral Pneumonia-161.png,True,False,299,299,L,128.8,f5f3759897bc653b3440202756e7132d
Viral Pneumonia-162.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-162.png,Viral Pneumonia/images/Viral Pneumonia-162.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-162.png,Viral Pneumonia/masks/Viral Pneumonia-162.png,True,False,299,299,L,152.69,d1d424761f829cfda3e8cf9aa41a60fc
Viral Pneumonia-163.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-163.png,Viral Pneumonia/images/Viral Pneumonia-163.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-163.png,Viral Pneumonia/masks/Viral Pneumonia-163.png,True,False,299,299,L,88.18,26afab207af4cb20ae68e6bfbadea498
Viral Pneumonia-164.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-164.png,Viral Pneumonia/images/Viral Pneumonia-164.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-164.png,Viral Pneumonia/masks/Viral Pneumonia-164.png,True,False,299,299,RGB,91.77,4d33c8fe11f5903a58e612a048f0f8e7
Viral Pneumonia-165.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-165.png,Viral Pneumonia/images/Viral Pneumonia-165.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-165.png,Viral Pneumonia/masks/Viral Pneumonia-165.png,True,False,299,299,L,76.78,0fe50b3832a2218bbdc6d8bc04e147d9
Viral Pneumonia-166.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-166.png,Viral Pneumonia/images/Viral Pneumonia-166.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-166.png,Viral Pneumonia/masks/Viral Pneumonia-166.png,True,False,299,299,L,137.62,167809497c57d8eb397568afd41ec7db
Viral Pneumonia-167.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-167.png,Viral Pneumonia/images/Viral Pneumonia-167.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-167.png,Viral Pneumonia/masks/Viral Pneumonia-167.png,True,False,299,299,L,83.48,2759c9f9014205e200be6535a97c0bf2
Viral Pneumonia-168.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-168.png,Viral Pneumonia/images/Viral Pneumonia-168.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-168.png,Viral Pneumonia/masks/Viral Pneumonia-168.png,True,False,299,299,L,107.13,9567dff091a03520a4159079ae06d99a
Viral Pneumonia-169.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-169.png,Viral Pneumonia/images/Viral Pneumonia-169.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-169.png,Viral Pneumonia/masks/Viral Pneumonia-169.png,True,False,299,299,L,113.39,9e0ba2c45f32e329003fbed2485f4ff3
Viral Pneumonia-17.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-17.png,Viral Pneumonia/images/Viral Pneumonia-17.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-17.png,Viral Pneumonia/masks/Viral Pneumonia-17.png,True,False,299,299,L,104.76,7b3cbb86de25cd0c0ad8ecfa9f082a09
Viral Pneumonia-170.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-170.png,Viral Pneumonia/images/Viral Pneumonia-170.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-170.png,Viral Pneumonia/masks/Viral Pneumonia-170.png,True,False,299,299,RGB,146.97,0fc0c1679f38c683055aab0825811985
Viral Pneumonia-171.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-171.png,Viral Pneumonia/images/Viral Pneumonia-171.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-171.png,Viral Pneumonia/masks/Viral Pneumonia-171.png,True,False,299,299,L,98.42,9807900bee0a64bbdb4b48780d129811
Viral Pneumonia-172.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-172.png,Viral Pneumonia/images/Viral Pneumonia-172.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-172.png,Viral Pneumonia/masks/Viral Pneumonia-172.png,True,False,299,299,L,148.65,56d169a71d4eef747285c782dfeade8f
Viral Pneumonia-173.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-173.png,Viral Pneumonia/images/Viral Pneumonia-173.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-173.png,Viral Pneumonia/masks/Viral Pneumonia-173.png,True,False,299,299,L,126.91,9d1eb835d1d07b716f29e3e79ebf297c
Viral Pneumonia-174.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-174.png,Viral Pneumonia/images/Viral Pneumonia-174.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-174.png,Viral Pneumonia/masks/Viral Pneumonia-174.png,True,False,299,299,L,120.22,237ac8ac9c829453a454c6a441bf70ae
Viral Pneumonia-175.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-175.png,Viral Pneumonia/images/Viral Pneumonia-175.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-175.png,Viral Pneumonia/masks/Viral Pneumonia-175.png,True,False,299,299,L,112.01,594f283f2f0956c9a769cf06e36b12d5
Viral Pneumonia-176.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-176.png,Viral Pneumonia/images/Viral Pneumonia-176.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-176.png,Viral Pneumonia/masks/Viral Pneumonia-176.png,True,False,299,299,L,122.61,b27a0894900ff4d3d3abbd5ff91893d2
Viral Pneumonia-177.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-177.png,Viral Pneumonia/images/Viral Pneumonia-177.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-177.png,Viral Pneumonia/masks/Viral Pneumonia-177.png,True,False,299,299,L,109.08,f4d6476a7f4abcce0a88cdce0dc1715b
Viral Pneumonia-178.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-178.png,Viral Pneumonia/images/Viral Pneumonia-178.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-178.png,Viral Pneumonia/masks/Viral Pneumonia-178.png,True,False,299,299,L,131.24,f441438e3930a8f5d43631078d02d60e
Viral Pneumonia-179.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-179.png,Viral Pneumonia/images/Viral Pneumonia-179.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-179.png,Viral Pneumonia/masks/Viral Pneumonia-179.png,True,False,299,299,L,133.77,e97c9891753bf76543c73b20203c1772
Viral Pneumonia-18.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-18.png,Viral Pneumonia/images/Viral Pneumonia-18.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-18.png,Viral Pneumonia/masks/Viral Pneumonia-18.png,True,False,299,299,L,100.2,1c5b33895cfb3d5481ce7a7cf95fae4e
Viral Pneumonia-180.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-180.png,Viral Pneumonia/images/Viral Pneumonia-180.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-180.png,Viral Pneumonia/masks/Viral Pneumonia-180.png,True,False,299,299,L,116.72,75da8c2be9a6d585093a79a362d0d11b
Viral Pneumonia-181.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-181.png,Viral Pneumonia/images/Viral Pneumonia-181.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-181.png,Viral Pneumonia/masks/Viral Pneumonia-181.png,True,False,299,299,RGB,136.51,ecd41cd1a8d06f4896bb29070b308f47
Viral Pneumonia-182.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-182.png,Viral Pneumonia/images/Viral Pneumonia-182.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-182.png,Viral Pneumonia/masks/Viral Pneumonia-182.png,True,False,299,299,L,136.31,5ca8b3006ec8476dcae0a643ff0c5454
Viral Pneumonia-183.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-183.png,Viral Pneumonia/images/Viral Pneumonia-183.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-183.png,Viral Pneumonia/masks/Viral Pneumonia-183.png,True,False,299,299,L,125.73,73bf52155f5c01d16a012e9eb8f2a162
Viral Pneumonia-184.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-184.png,Viral Pneumonia/images/Viral Pneumonia-184.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-184.png,Viral Pneumonia/masks/Viral Pneumonia-184.png,True,False,299,299,RGB,134.43,bbc28322576a9126a3ce6ef1f97b0a09
Viral Pneumonia-185.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-185.png,Viral Pneumonia/images/Viral Pneumonia-185.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-185.png,Viral Pneumonia/masks/Viral Pneumonia-185.png,True,False,299,299,L,145.98,bb8d4cf604f3e537d9d394d21d30fdab
Viral Pneumonia-186.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-186.png,Viral Pneumonia/images/Viral Pneumonia-186.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-186.png,Viral Pneumonia/masks/Viral Pneumonia-186.png,True,False,299,299,L,130.57,824e9be1b2d39826643e4bc89e6a1c27
Viral Pneumonia-187.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-187.png,Viral Pneumonia/images/Viral Pneumonia-187.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-187.png,Viral Pneumonia/masks/Viral Pneumonia-187.png,True,False,299,299,L,127.17,c16020833227cd927d28a113fd4ea842
Viral Pneumonia-188.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-188.png,Viral Pneumonia/images/Viral Pneumonia-188.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-188.png,Viral Pneumonia/masks/Viral Pneumonia-188.png,True,False,299,299,L,130.56,f6566061c59fb32e9aa1d0b6ef61c692
Viral Pneumonia-189.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-189.png,Viral Pneumonia/images/Viral Pneumonia-189.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-189.png,Viral Pneumonia/masks/Viral Pneumonia-189.png,True,False,299,299,L,121.21,515c1020190e735e14403d132270fb8b
Viral Pneumonia-19.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-19.png,Viral Pneumonia/images/Viral Pneumonia-19.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-19.png,Viral Pneumonia/masks/Viral Pneumonia-19.png,True,False,299,299,L,156.22,ead647bd97f54084e1936936ffc32c5e
Viral Pneumonia-190.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-190.png,Viral Pneumonia/images/Viral Pneumonia-190.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-190.png,Viral Pneumonia/masks/Viral Pneumonia-190.png,True,False,299,299,L,141.43,0adc56ab3a690bb8ee98b7a633eb11af
Viral Pneumonia-191.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-191.png,Viral Pneumonia/images/Viral Pneumonia-191.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-191.png,Viral Pneumonia/masks/Viral Pneumonia-191.png,True,False,299,299,RGB,125.29,6cfb6386609423631e3fe1bda90a7fe0
Viral Pneumonia-192.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-192.png,Viral Pneumonia/images/Viral Pneumonia-192.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-192.png,Viral Pneumonia/masks/Viral Pneumonia-192.png,True,False,299,299,L,128.98,278e6539e3617e83f71f8435af585c0d
Viral Pneumonia-193.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-193.png,Viral Pneumonia/images/Viral Pneumonia-193.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-193.png,Viral Pneumonia/masks/Viral Pneumonia-193.png,True,False,299,299,L,136.57,0700c377a3a670ea4aad4d40d23992aa
Viral Pneumonia-194.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-194.png,Viral Pneumonia/images/Viral Pneumonia-194.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-194.png,Viral Pneumonia/masks/Viral Pneumonia-194.png,True,False,299,299,L,130.26,86084f5ad434df3f0770407f8898f478
Viral Pneumonia-195.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-195.png,Viral Pneumonia/images/Viral Pneumonia-195.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-195.png,Viral Pneumonia/masks/Viral Pneumonia-195.png,True,False,299,299,L,129.83,38727703bedb23016990d7993337a8d8
Viral Pneumonia-196.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-196.png,Viral Pneumonia/images/Viral Pneumonia-196.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-196.png,Viral Pneumonia/masks/Viral Pneumonia-196.png,True,False,299,299,RGB,142.4,281a34e341d81d1ca958f490d955ec8e
Viral Pneumonia-197.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-197.png,Viral Pneumonia/images/Viral Pneumonia-197.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-197.png,Viral Pneumonia/masks/Viral Pneumonia-197.png,True,False,299,299,L,145.7,efe0b1b86dad8f0a2d9a01da3b1be830
Viral Pneumonia-198.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-198.png,Viral Pneumonia/images/Viral Pneumonia-198.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-198.png,Viral Pneumonia/masks/Viral Pneumonia-198.png,True,False,299,299,L,129.08,4377dd7c393f63396959b5d1b7851469
Viral Pneumonia-199.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-199.png,Viral Pneumonia/images/Viral Pneumonia-199.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-199.png,Viral Pneumonia/masks/Viral Pneumonia-199.png,True,False,299,299,L,146.24,de03c1c4efd89058bbcae7c1245ef75e
Viral Pneumonia-2.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-2.png,Viral Pneumonia/images/Viral Pneumonia-2.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-2.png,Viral Pneumonia/masks/Viral Pneumonia-2.png,True,False,299,299,L,128.86,5626ee812c94c7bf3e16753c9bd887ce
Viral Pneumonia-20.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-20.png,Viral Pneumonia/images/Viral Pneumonia-20.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-20.png,Viral Pneumonia/masks/Viral Pneumonia-20.png,True,False,299,299,L,127.79,35601e7fdd0a6cf09a3eb0e0e4d9b90d
Viral Pneumonia-200.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-200.png,Viral Pneumonia/images/Viral Pneumonia-200.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-200.png,Viral Pneumonia/masks/Viral Pneumonia-200.png,True,False,299,299,L,131.33,0a0f0e66756eadfae05b726673f090bf
Viral Pneumonia-201.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-201.png,Viral Pneumonia/images/Viral Pneumonia-201.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-201.png,Viral Pneumonia/masks/Viral Pneumonia-201.png,True,False,299,299,RGB,155.1,121094bb232e08257049100befca3d5e
Viral Pneumonia-202.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-202.png,Viral Pneumonia/images/Viral Pneumonia-202.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-202.png,Viral Pneumonia/masks/Viral Pneumonia-202.png,True,False,299,299,RGB,144.41,a11831e98ede9112a1eb83d79361908c
Viral Pneumonia-203.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-203.png,Viral Pneumonia/images/Viral Pneumonia-203.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-203.png,Viral Pneumonia/masks/Viral Pneumonia-203.png,True,False,299,299,RGB,104.05,36f486436dbe0977046587ff9f4989a3
Viral Pneumonia-204.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-204.png,Viral Pneumonia/images/Viral Pneumonia-204.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-204.png,Viral Pneumonia/masks/Viral Pneumonia-204.png,True,False,299,299,RGB,134.62,e9463c6e4fcd773cbaa6c92599f308de
Viral Pneumonia-205.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-205.png,Viral Pneumonia/images/Viral Pneumonia-205.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-205.png,Viral Pneumonia/masks/Viral Pneumonia-205.png,True,False,299,299,RGB,118.61,1c0061cef331d551648b9f677c2f3bbe
Viral Pneumonia-206.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-206.png,Viral Pneumonia/images/Viral Pneumonia-206.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-206.png,Viral Pneumonia/masks/Viral Pneumonia-206.png,True,False,299,299,L,123.98,f40973413ccfb0ba07a3fd419a4af079
Viral Pneumonia-207.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-207.png,Viral Pneumonia/images/Viral Pneumonia-207.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-207.png,Viral Pneumonia/masks/Viral Pneumonia-207.png,True,False,299,299,L,136.64,fddc8e440d0f6389c6ea48ebbd0c3bc0
Viral Pneumonia-208.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-208.png,Viral Pneumonia/images/Viral Pneumonia-208.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-208.png,Viral Pneumonia/masks/Viral Pneumonia-208.png,True,False,299,299,RGB,122.65,db753afce2a1a99336a598b6ffd05823
Viral Pneumonia-209.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-209.png,Viral Pneumonia/images/Viral Pneumonia-209.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-209.png,Viral Pneumonia/masks/Viral Pneumonia-209.png,True,False,299,299,L,126.36,00d453cc731217bc914b232e2f1cdc3e
Viral Pneumonia-21.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-21.png,Viral Pneumonia/images/Viral Pneumonia-21.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-21.png,Viral Pneumonia/masks/Viral Pneumonia-21.png,True,False,299,299,L,109.0,8db63b33f43bd2001d85d418cb12bda3
Viral Pneumonia-210.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-210.png,Viral Pneumonia/images/Viral Pneumonia-210.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-210.png,Viral Pneumonia/masks/Viral Pneumonia-210.png,True,False,299,299,L,131.83,3729a46cb6d29c72eed5198202ffb7fc
Viral Pneumonia-211.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-211.png,Viral Pneumonia/images/Viral Pneumonia-211.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-211.png,Viral Pneumonia/masks/Viral Pneumonia-211.png,True,False,299,299,L,124.26,4577877513eafe5ca0b168fc0e811c0b
Viral Pneumonia-212.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-212.png,Viral Pneumonia/images/Viral Pneumonia-212.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-212.png,Viral Pneumonia/masks/Viral Pneumonia-212.png,True,False,299,299,L,154.17,4b6739bdf36054da4d5e0e4095bc4e3a
Viral Pneumonia-213.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-213.png,Viral Pneumonia/images/Viral Pneumonia-213.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-213.png,Viral Pneumonia/masks/Viral Pneumonia-213.png,True,False,299,299,L,144.93,92c1049b35b12bae8ea5835bc52b62b7
Viral Pneumonia-214.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-214.png,Viral Pneumonia/images/Viral Pneumonia-214.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-214.png,Viral Pneumonia/masks/Viral Pneumonia-214.png,True,False,299,299,L,130.32,084105723943df132da3beb9e69822ca
Viral Pneumonia-215.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-215.png,Viral Pneumonia/images/Viral Pneumonia-215.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-215.png,Viral Pneumonia/masks/Viral Pneumonia-215.png,True,False,299,299,L,93.01,2fafe63484791fc2f8e63aa37a8410c2
Viral Pneumonia-216.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-216.png,Viral Pneumonia/images/Viral Pneumonia-216.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-216.png,Viral Pneumonia/masks/Viral Pneumonia-216.png,True,False,299,299,L,134.28,1a3619156dcd36c3186dcd0d44997161
Viral Pneumonia-217.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-217.png,Viral Pneumonia/images/Viral Pneumonia-217.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-217.png,Viral Pneumonia/masks/Viral Pneumonia-217.png,True,False,299,299,L,128.05,de50b569f5447931e9beddda926f33c7
Viral Pneumonia-218.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-218.png,Viral Pneumonia/images/Viral Pneumonia-218.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-218.png,Viral Pneumonia/masks/Viral Pneumonia-218.png,True,False,299,299,L,129.77,5c71696d7fda348bd352ae22b8145da5
Viral Pneumonia-219.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-219.png,Viral Pneumonia/images/Viral Pneumonia-219.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-219.png,Viral Pneumonia/masks/Viral Pneumonia-219.png,True,False,299,299,L,122.98,0b24c2681ce5449a342f44577f1c5648
Viral Pneumonia-22.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-22.png,Viral Pneumonia/images/Viral Pneumonia-22.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-22.png,Viral Pneumonia/masks/Viral Pneumonia-22.png,True,False,299,299,L,112.46,038ff29e52c8124c8189f966963ddf4d
Viral Pneumonia-220.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-220.png,Viral Pneumonia/images/Viral Pneumonia-220.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-220.png,Viral Pneumonia/masks/Viral Pneumonia-220.png,True,False,299,299,RGB,124.24,bae595880046d28d63783198a572ff76
Viral Pneumonia-221.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-221.png,Viral Pneumonia/images/Viral Pneumonia-221.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-221.png,Viral Pneumonia/masks/Viral Pneumonia-221.png,True,False,299,299,L,112.15,47f31ed0b0303dfe19db3633e0677411
Viral Pneumonia-222.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-222.png,Viral Pneumonia/images/Viral Pneumonia-222.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-222.png,Viral Pneumonia/masks/Viral Pneumonia-222.png,True,False,299,299,RGB,172.32,75d691986ba416fc6a3be258b019a02a
Viral Pneumonia-223.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-223.png,Viral Pneumonia/images/Viral Pneumonia-223.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-223.png,Viral Pneumonia/masks/Viral Pneumonia-223.png,True,False,299,299,L,142.24,500358be59e129ae8411775582feb59e
Viral Pneumonia-224.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-224.png,Viral Pneumonia/images/Viral Pneumonia-224.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-224.png,Viral Pneumonia/masks/Viral Pneumonia-224.png,True,False,299,299,L,128.64,4251b018e5b073ee5b9445e2f1ac580c
Viral Pneumonia-225.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-225.png,Viral Pneumonia/images/Viral Pneumonia-225.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-225.png,Viral Pneumonia/masks/Viral Pneumonia-225.png,True,False,299,299,L,142.66,5b92c4818a1c942d0870fe657c94fbba
Viral Pneumonia-226.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-226.png,Viral Pneumonia/images/Viral Pneumonia-226.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-226.png,Viral Pneumonia/masks/Viral Pneumonia-226.png,True,False,299,299,RGB,126.82,d989e1823aea6723e46b7f69e875fcbf
Viral Pneumonia-227.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-227.png,Viral Pneumonia/images/Viral Pneumonia-227.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-227.png,Viral Pneumonia/masks/Viral Pneumonia-227.png,True,False,299,299,RGB,143.59,fa5fcb678ecfac95bbfe7290aed00ae6
Viral Pneumonia-228.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-228.png,Viral Pneumonia/images/Viral Pneumonia-228.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-228.png,Viral Pneumonia/masks/Viral Pneumonia-228.png,True,False,299,299,RGB,121.18,47c5687b64eefc001d7da6daf70c06e5
Viral Pneumonia-229.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-229.png,Viral Pneumonia/images/Viral Pneumonia-229.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-229.png,Viral Pneumonia/masks/Viral Pneumonia-229.png,True,False,299,299,L,110.0,f1f5fe28a7d73b7a1b97b6f1e36d8202
Viral Pneumonia-23.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-23.png,Viral Pneumonia/images/Viral Pneumonia-23.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-23.png,Viral Pneumonia/masks/Viral Pneumonia-23.png,True,False,299,299,L,146.21,e0a83bfabf2821eff70df4526c15f20e
Viral Pneumonia-230.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-230.png,Viral Pneumonia/images/Viral Pneumonia-230.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-230.png,Viral Pneumonia/masks/Viral Pneumonia-230.png,True,False,299,299,L,128.66,27c2cdcf3c58d26448eb82b512086e0d
Viral Pneumonia-231.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-231.png,Viral Pneumonia/images/Viral Pneumonia-231.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-231.png,Viral Pneumonia/masks/Viral Pneumonia-231.png,True,False,299,299,L,141.65,6b39af4006a444926a01b2438b4eec8e
Viral Pneumonia-232.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-232.png,Viral Pneumonia/images/Viral Pneumonia-232.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-232.png,Viral Pneumonia/masks/Viral Pneumonia-232.png,True,False,299,299,L,145.55,4853571c3412ae50372ffef2c560c9b5
Viral Pneumonia-233.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-233.png,Viral Pneumonia/images/Viral Pneumonia-233.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-233.png,Viral Pneumonia/masks/Viral Pneumonia-233.png,True,False,299,299,L,135.71,3748b3f3d071ca24a77d9b5ab403c319
Viral Pneumonia-234.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-234.png,Viral Pneumonia/images/Viral Pneumonia-234.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-234.png,Viral Pneumonia/masks/Viral Pneumonia-234.png,True,False,299,299,RGB,149.79,9b33b516c6d95101e8fe7a2fc63ad387
Viral Pneumonia-235.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-235.png,Viral Pneumonia/images/Viral Pneumonia-235.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-235.png,Viral Pneumonia/masks/Viral Pneumonia-235.png,True,False,299,299,L,118.87,be73a8defd86d8df3e9141d4319691ed
Viral Pneumonia-236.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-236.png,Viral Pneumonia/images/Viral Pneumonia-236.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-236.png,Viral Pneumonia/masks/Viral Pneumonia-236.png,True,False,299,299,L,121.77,bde5e618f2ffcdb9ed3b41e255a70ef7
Viral Pneumonia-237.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-237.png,Viral Pneumonia/images/Viral Pneumonia-237.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-237.png,Viral Pneumonia/masks/Viral Pneumonia-237.png,True,False,299,299,L,69.22,211d1c7acd1606863b49be71cf6ab709
Viral Pneumonia-238.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-238.png,Viral Pneumonia/images/Viral Pneumonia-238.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-238.png,Viral Pneumonia/masks/Viral Pneumonia-238.png,True,False,299,299,L,126.58,fa5b27f387b84464f64ff44607c4c0b3
Viral Pneumonia-239.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-239.png,Viral Pneumonia/images/Viral Pneumonia-239.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-239.png,Viral Pneumonia/masks/Viral Pneumonia-239.png,True,False,299,299,L,142.79,50ba35d47faa6b5c1f67c3799cc7d2f6
Viral Pneumonia-24.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-24.png,Viral Pneumonia/images/Viral Pneumonia-24.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-24.png,Viral Pneumonia/masks/Viral Pneumonia-24.png,True,False,299,299,L,146.22,66b34bee9e62846a36571586e81e08a1
Viral Pneumonia-240.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-240.png,Viral Pneumonia/images/Viral Pneumonia-240.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-240.png,Viral Pneumonia/masks/Viral Pneumonia-240.png,True,False,299,299,L,118.57,28e3c50c4dc99cc1a1c6fa0e2d093d71
Viral Pneumonia-241.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-241.png,Viral Pneumonia/images/Viral Pneumonia-241.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-241.png,Viral Pneumonia/masks/Viral Pneumonia-241.png,True,False,299,299,L,132.07,f82460471d8f6cc3bc2538cf203c71f6
Viral Pneumonia-242.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-242.png,Viral Pneumonia/images/Viral Pneumonia-242.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-242.png,Viral Pneumonia/masks/Viral Pneumonia-242.png,True,False,299,299,L,134.32,3e9308ed064657eb158820e626897e61
Viral Pneumonia-243.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-243.png,Viral Pneumonia/images/Viral Pneumonia-243.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-243.png,Viral Pneumonia/masks/Viral Pneumonia-243.png,True,False,299,299,L,130.03,5ed92f712c9bda8762e2aa18e3f64f61
Viral Pneumonia-244.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-244.png,Viral Pneumonia/images/Viral Pneumonia-244.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-244.png,Viral Pneumonia/masks/Viral Pneumonia-244.png,True,False,299,299,L,102.27,254a1afef84d3c1d48700d13cc4de1f0
Viral Pneumonia-245.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-245.png,Viral Pneumonia/images/Viral Pneumonia-245.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-245.png,Viral Pneumonia/masks/Viral Pneumonia-245.png,True,False,299,299,L,120.54,c2d03b3e2cbb7e7e0b68a42dbd461fec
Viral Pneumonia-246.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-246.png,Viral Pneumonia/images/Viral Pneumonia-246.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-246.png,Viral Pneumonia/masks/Viral Pneumonia-246.png,True,False,299,299,L,130.79,39bd85eb4af350aedb60c3641d737e21
Viral Pneumonia-247.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-247.png,Viral Pneumonia/images/Viral Pneumonia-247.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-247.png,Viral Pneumonia/masks/Viral Pneumonia-247.png,True,False,299,299,RGB,121.18,aa09b17bee2a2175311b1856ec917469
Viral Pneumonia-248.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-248.png,Viral Pneumonia/images/Viral Pneumonia-248.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-248.png,Viral Pneumonia/masks/Viral Pneumonia-248.png,True,False,299,299,L,107.67,712612abf127659126d6ab7ae6d7f7f7
Viral Pneumonia-249.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-249.png,Viral Pneumonia/images/Viral Pneumonia-249.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-249.png,Viral Pneumonia/masks/Viral Pneumonia-249.png,True,False,299,299,L,123.29,60cf0049b29d4ee1c0743b969a4b2130
Viral Pneumonia-25.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-25.png,Viral Pneumonia/images/Viral Pneumonia-25.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-25.png,Viral Pneumonia/masks/Viral Pneumonia-25.png,True,False,299,299,L,85.28,77926d4a3482d508490ee8fdb624be32
Viral Pneumonia-250.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-250.png,Viral Pneumonia/images/Viral Pneumonia-250.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-250.png,Viral Pneumonia/masks/Viral Pneumonia-250.png,True,False,299,299,L,123.29,60cf0049b29d4ee1c0743b969a4b2130
Viral Pneumonia-251.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-251.png,Viral Pneumonia/images/Viral Pneumonia-251.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-251.png,Viral Pneumonia/masks/Viral Pneumonia-251.png,True,False,299,299,L,137.19,660c1ca69c2bf4ee2848fedcaee8fb1b
Viral Pneumonia-252.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-252.png,Viral Pneumonia/images/Viral Pneumonia-252.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-252.png,Viral Pneumonia/masks/Viral Pneumonia-252.png,True,False,299,299,L,119.7,79b2669a52a3dde6a32a9f6787a90cf6
Viral Pneumonia-253.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-253.png,Viral Pneumonia/images/Viral Pneumonia-253.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-253.png,Viral Pneumonia/masks/Viral Pneumonia-253.png,True,False,299,299,L,132.05,685cbc89300dd95b4e4d4d01057bc126
Viral Pneumonia-254.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-254.png,Viral Pneumonia/images/Viral Pneumonia-254.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-254.png,Viral Pneumonia/masks/Viral Pneumonia-254.png,True,False,299,299,L,163.92,38ddf0ba2d0ced20a565905355f2c2de
Viral Pneumonia-255.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-255.png,Viral Pneumonia/images/Viral Pneumonia-255.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-255.png,Viral Pneumonia/masks/Viral Pneumonia-255.png,True,False,299,299,L,99.2,1872fcad19a3595b9eb8f8b54437a5be
Viral Pneumonia-256.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-256.png,Viral Pneumonia/images/Viral Pneumonia-256.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-256.png,Viral Pneumonia/masks/Viral Pneumonia-256.png,True,False,299,299,L,118.92,bbb95c5ef82dee6f95dd1909525091a4
Viral Pneumonia-257.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-257.png,Viral Pneumonia/images/Viral Pneumonia-257.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-257.png,Viral Pneumonia/masks/Viral Pneumonia-257.png,True,False,299,299,L,122.25,96dfb72f7eec6683aa242379b641ce3f
Viral Pneumonia-258.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-258.png,Viral Pneumonia/images/Viral Pneumonia-258.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-258.png,Viral Pneumonia/masks/Viral Pneumonia-258.png,True,False,299,299,L,128.25,f0b759cca6e0b3a9f66797fff736b72e
Viral Pneumonia-259.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-259.png,Viral Pneumonia/images/Viral Pneumonia-259.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-259.png,Viral Pneumonia/masks/Viral Pneumonia-259.png,True,False,299,299,RGB,146.79,2fa36dc21b7d17708d600151cf88c2ae
Viral Pneumonia-26.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-26.png,Viral Pneumonia/images/Viral Pneumonia-26.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-26.png,Viral Pneumonia/masks/Viral Pneumonia-26.png,True,False,299,299,L,142.5,4287e37b3c3c28777de04f2c883c3115
Viral Pneumonia-260.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-260.png,Viral Pneumonia/images/Viral Pneumonia-260.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-260.png,Viral Pneumonia/masks/Viral Pneumonia-260.png,True,False,299,299,L,121.05,0ee33608435c586a64da7d21cadb6452
Viral Pneumonia-261.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-261.png,Viral Pneumonia/images/Viral Pneumonia-261.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-261.png,Viral Pneumonia/masks/Viral Pneumonia-261.png,True,False,299,299,L,128.58,71298817af2d7805499d3681d26d5a79
Viral Pneumonia-262.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-262.png,Viral Pneumonia/images/Viral Pneumonia-262.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-262.png,Viral Pneumonia/masks/Viral Pneumonia-262.png,True,False,299,299,L,132.53,be0c3a6d7e4ba7adc61aa08f8bc33f34
Viral Pneumonia-263.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-263.png,Viral Pneumonia/images/Viral Pneumonia-263.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-263.png,Viral Pneumonia/masks/Viral Pneumonia-263.png,True,False,299,299,L,126.42,80866503419a97da281aa5d7ff1713c5
Viral Pneumonia-264.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-264.png,Viral Pneumonia/images/Viral Pneumonia-264.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-264.png,Viral Pneumonia/masks/Viral Pneumonia-264.png,True,False,299,299,L,117.41,cd4783cbca4b4c74c90a44a13d31342f
Viral Pneumonia-265.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-265.png,Viral Pneumonia/images/Viral Pneumonia-265.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-265.png,Viral Pneumonia/masks/Viral Pneumonia-265.png,True,False,299,299,L,170.56,107d4a9fad41f7f4e9c8171c84b91ca2
Viral Pneumonia-266.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-266.png,Viral Pneumonia/images/Viral Pneumonia-266.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-266.png,Viral Pneumonia/masks/Viral Pneumonia-266.png,True,False,299,299,L,133.35,b2d8f0f5eca04630f27f4665d1a51263
Viral Pneumonia-267.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-267.png,Viral Pneumonia/images/Viral Pneumonia-267.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-267.png,Viral Pneumonia/masks/Viral Pneumonia-267.png,True,False,299,299,L,122.14,763338a227cfcd2142abc6f1810203c3
Viral Pneumonia-268.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-268.png,Viral Pneumonia/images/Viral Pneumonia-268.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-268.png,Viral Pneumonia/masks/Viral Pneumonia-268.png,True,False,299,299,L,156.3,638fdff1dd073a9d9e07e36cb6a18269
Viral Pneumonia-269.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-269.png,Viral Pneumonia/images/Viral Pneumonia-269.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-269.png,Viral Pneumonia/masks/Viral Pneumonia-269.png,True,False,299,299,L,142.64,627941bc094e56bc18ff991401246701
Viral Pneumonia-27.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-27.png,Viral Pneumonia/images/Viral Pneumonia-27.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-27.png,Viral Pneumonia/masks/Viral Pneumonia-27.png,True,False,299,299,L,107.37,bd9b605072f09c88abea416229863f97
Viral Pneumonia-270.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-270.png,Viral Pneumonia/images/Viral Pneumonia-270.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-270.png,Viral Pneumonia/masks/Viral Pneumonia-270.png,True,False,299,299,L,127.85,e07efdb7048041bd30b105fdbbfb6824
Viral Pneumonia-271.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-271.png,Viral Pneumonia/images/Viral Pneumonia-271.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-271.png,Viral Pneumonia/masks/Viral Pneumonia-271.png,True,False,299,299,L,125.56,85752e8ede478c832e677ed4315a3c41
Viral Pneumonia-272.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-272.png,Viral Pneumonia/images/Viral Pneumonia-272.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-272.png,Viral Pneumonia/masks/Viral Pneumonia-272.png,True,False,299,299,L,136.55,ae9a21a3f7f8e57bfc4a509bb1780fa5
Viral Pneumonia-273.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-273.png,Viral Pneumonia/images/Viral Pneumonia-273.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-273.png,Viral Pneumonia/masks/Viral Pneumonia-273.png,True,False,299,299,L,132.41,51c1b800d37979c25a153bd2eb6e3ea9
Viral Pneumonia-274.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-274.png,Viral Pneumonia/images/Viral Pneumonia-274.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-274.png,Viral Pneumonia/masks/Viral Pneumonia-274.png,True,False,299,299,L,115.88,8a29c5f0b3b106084d55b93a848ab07d
Viral Pneumonia-275.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-275.png,Viral Pneumonia/images/Viral Pneumonia-275.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-275.png,Viral Pneumonia/masks/Viral Pneumonia-275.png,True,False,299,299,L,127.07,03aa36dc9b494666aa667625dd4a9ec8
Viral Pneumonia-276.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-276.png,Viral Pneumonia/images/Viral Pneumonia-276.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-276.png,Viral Pneumonia/masks/Viral Pneumonia-276.png,True,False,299,299,L,142.3,2bd1e7ecc90f4a89886ae5ab70225d58
Viral Pneumonia-277.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-277.png,Viral Pneumonia/images/Viral Pneumonia-277.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-277.png,Viral Pneumonia/masks/Viral Pneumonia-277.png,True,False,299,299,L,127.32,a407b3f6fa1c3136bd1d452f8612c62f
Viral Pneumonia-278.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-278.png,Viral Pneumonia/images/Viral Pneumonia-278.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-278.png,Viral Pneumonia/masks/Viral Pneumonia-278.png,True,False,299,299,L,147.68,ccf70e356ecfe417ed38b250e7fb2710
Viral Pneumonia-279.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-279.png,Viral Pneumonia/images/Viral Pneumonia-279.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-279.png,Viral Pneumonia/masks/Viral Pneumonia-279.png,True,False,299,299,L,129.28,e6f66133f966b90320e4db28a837c1a1
Viral Pneumonia-28.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-28.png,Viral Pneumonia/images/Viral Pneumonia-28.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-28.png,Viral Pneumonia/masks/Viral Pneumonia-28.png,True,False,299,299,L,115.93,4dbeddd464bf8b718def5ee18e87fce0
Viral Pneumonia-280.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-280.png,Viral Pneumonia/images/Viral Pneumonia-280.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-280.png,Viral Pneumonia/masks/Viral Pneumonia-280.png,True,False,299,299,L,153.7,1b2d39327879e1e40c147a56174b216a
Viral Pneumonia-281.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-281.png,Viral Pneumonia/images/Viral Pneumonia-281.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-281.png,Viral Pneumonia/masks/Viral Pneumonia-281.png,True,False,299,299,L,106.02,9cc0165360e9fff125b4464981013e85
Viral Pneumonia-282.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-282.png,Viral Pneumonia/images/Viral Pneumonia-282.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-282.png,Viral Pneumonia/masks/Viral Pneumonia-282.png,True,False,299,299,RGB,150.2,06919e538d6b67e293ff8b44c30fcf2d
Viral Pneumonia-283.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-283.png,Viral Pneumonia/images/Viral Pneumonia-283.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-283.png,Viral Pneumonia/masks/Viral Pneumonia-283.png,True,False,299,299,L,127.03,d009816ebc19b15304f3026da6fbedbe
Viral Pneumonia-284.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-284.png,Viral Pneumonia/images/Viral Pneumonia-284.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-284.png,Viral Pneumonia/masks/Viral Pneumonia-284.png,True,False,299,299,L,141.1,d897ebb8f8b5d6b48b2d785cb3039970
Viral Pneumonia-285.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-285.png,Viral Pneumonia/images/Viral Pneumonia-285.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-285.png,Viral Pneumonia/masks/Viral Pneumonia-285.png,True,False,299,299,L,114.92,2596999425488ddfa8eba62e6abf37db
Viral Pneumonia-286.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-286.png,Viral Pneumonia/images/Viral Pneumonia-286.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-286.png,Viral Pneumonia/masks/Viral Pneumonia-286.png,True,False,299,299,L,147.65,a63d5f8226e0d02a3bc47a7d7e06dc4e
Viral Pneumonia-287.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-287.png,Viral Pneumonia/images/Viral Pneumonia-287.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-287.png,Viral Pneumonia/masks/Viral Pneumonia-287.png,True,False,299,299,L,132.05,a08cd4bbb6014d5617e84ad22b0cd22a
Viral Pneumonia-288.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-288.png,Viral Pneumonia/images/Viral Pneumonia-288.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-288.png,Viral Pneumonia/masks/Viral Pneumonia-288.png,True,False,299,299,L,127.68,81bd6fceb125126c29f2de1131e7919e
Viral Pneumonia-289.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-289.png,Viral Pneumonia/images/Viral Pneumonia-289.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-289.png,Viral Pneumonia/masks/Viral Pneumonia-289.png,True,False,299,299,L,107.32,7ef15a1307b604c88a3c6146c477da36
Viral Pneumonia-29.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-29.png,Viral Pneumonia/images/Viral Pneumonia-29.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-29.png,Viral Pneumonia/masks/Viral Pneumonia-29.png,True,False,299,299,L,106.47,4fde48a1fe8fbb132f5731884863ef29
Viral Pneumonia-290.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-290.png,Viral Pneumonia/images/Viral Pneumonia-290.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-290.png,Viral Pneumonia/masks/Viral Pneumonia-290.png,True,False,299,299,L,100.37,9a339fdf56feb30651a4d1187269153d
Viral Pneumonia-291.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-291.png,Viral Pneumonia/images/Viral Pneumonia-291.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-291.png,Viral Pneumonia/masks/Viral Pneumonia-291.png,True,False,299,299,L,128.4,f077a44e7fc807c5971984b6060bd5bb
Viral Pneumonia-292.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-292.png,Viral Pneumonia/images/Viral Pneumonia-292.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-292.png,Viral Pneumonia/masks/Viral Pneumonia-292.png,True,False,299,299,L,132.51,87ae949f1673e09480e354ea9520686f
Viral Pneumonia-293.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-293.png,Viral Pneumonia/images/Viral Pneumonia-293.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-293.png,Viral Pneumonia/masks/Viral Pneumonia-293.png,True,False,299,299,L,129.23,28f48c4a62e5b37ab670d94a56b69b99
Viral Pneumonia-294.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-294.png,Viral Pneumonia/images/Viral Pneumonia-294.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-294.png,Viral Pneumonia/masks/Viral Pneumonia-294.png,True,False,299,299,L,95.98,a50d6fe17090c2a3741d2627446374a1
Viral Pneumonia-295.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-295.png,Viral Pneumonia/images/Viral Pneumonia-295.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-295.png,Viral Pneumonia/masks/Viral Pneumonia-295.png,True,False,299,299,L,95.98,a50d6fe17090c2a3741d2627446374a1
Viral Pneumonia-296.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-296.png,Viral Pneumonia/images/Viral Pneumonia-296.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-296.png,Viral Pneumonia/masks/Viral Pneumonia-296.png,True,False,299,299,L,141.23,e2edf92b462b4ccb974e37bb10b3dac1
Viral Pneumonia-297.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-297.png,Viral Pneumonia/images/Viral Pneumonia-297.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-297.png,Viral Pneumonia/masks/Viral Pneumonia-297.png,True,False,299,299,L,136.83,bc6b761cd60b13438b2d6670339b5bfd
Viral Pneumonia-298.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-298.png,Viral Pneumonia/images/Viral Pneumonia-298.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-298.png,Viral Pneumonia/masks/Viral Pneumonia-298.png,True,False,299,299,RGB,130.04,45c27c584bfacc12015071853bcbc32a
Viral Pneumonia-299.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-299.png,Viral Pneumonia/images/Viral Pneumonia-299.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-299.png,Viral Pneumonia/masks/Viral Pneumonia-299.png,True,False,299,299,L,131.58,5d9b89f63da2b8cec62770ba30dfda2b
Viral Pneumonia-3.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-3.png,Viral Pneumonia/images/Viral Pneumonia-3.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-3.png,Viral Pneumonia/masks/Viral Pneumonia-3.png,True,False,299,299,L,125.17,9bd60761b0f92f3978db32c93d5cd74f
Viral Pneumonia-30.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-30.png,Viral Pneumonia/images/Viral Pneumonia-30.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-30.png,Viral Pneumonia/masks/Viral Pneumonia-30.png,True,False,299,299,L,137.45,6472ddd09740f1277876493e7063bd77
Viral Pneumonia-300.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-300.png,Viral Pneumonia/images/Viral Pneumonia-300.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-300.png,Viral Pneumonia/masks/Viral Pneumonia-300.png,True,False,299,299,RGB,123.56,46051e3902378d1003e1165d0d8a8e86
Viral Pneumonia-301.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-301.png,Viral Pneumonia/images/Viral Pneumonia-301.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-301.png,Viral Pneumonia/masks/Viral Pneumonia-301.png,True,False,299,299,RGB,118.76,59bf74b040944a23eec2ea1cc48af767
Viral Pneumonia-302.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-302.png,Viral Pneumonia/images/Viral Pneumonia-302.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-302.png,Viral Pneumonia/masks/Viral Pneumonia-302.png,True,False,299,299,RGB,143.68,71672b8f7fdb9948e5f9e882c359ad04
Viral Pneumonia-303.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-303.png,Viral Pneumonia/images/Viral Pneumonia-303.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-303.png,Viral Pneumonia/masks/Viral Pneumonia-303.png,True,False,299,299,L,117.5,6c4a823374004ab87b6f34d25a2cf4df
Viral Pneumonia-304.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-304.png,Viral Pneumonia/images/Viral Pneumonia-304.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-304.png,Viral Pneumonia/masks/Viral Pneumonia-304.png,True,False,299,299,L,127.07,ba74ab09c164bafec16c376a1379f690
Viral Pneumonia-305.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-305.png,Viral Pneumonia/images/Viral Pneumonia-305.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-305.png,Viral Pneumonia/masks/Viral Pneumonia-305.png,True,False,299,299,L,134.08,8a1ca1ef211e253d4dd01748a3534afe
Viral Pneumonia-306.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-306.png,Viral Pneumonia/images/Viral Pneumonia-306.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-306.png,Viral Pneumonia/masks/Viral Pneumonia-306.png,True,False,299,299,L,141.1,f29f2c7553ba041659b56fbb160c5d9a
Viral Pneumonia-307.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-307.png,Viral Pneumonia/images/Viral Pneumonia-307.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-307.png,Viral Pneumonia/masks/Viral Pneumonia-307.png,True,False,299,299,L,117.92,9fb7d7bb79673d721063aa204ff8cacd
Viral Pneumonia-308.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-308.png,Viral Pneumonia/images/Viral Pneumonia-308.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-308.png,Viral Pneumonia/masks/Viral Pneumonia-308.png,True,False,299,299,L,121.3,0d94ffcb64de2c0aa7a7a5b436094288
Viral Pneumonia-309.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-309.png,Viral Pneumonia/images/Viral Pneumonia-309.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-309.png,Viral Pneumonia/masks/Viral Pneumonia-309.png,True,False,299,299,RGB,125.3,3f4e1a663e309515ac58951eb55361e5
Viral Pneumonia-31.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-31.png,Viral Pneumonia/images/Viral Pneumonia-31.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-31.png,Viral Pneumonia/masks/Viral Pneumonia-31.png,True,False,299,299,L,73.5,aa9548886626c18ff5d8bbf154ead47d
Viral Pneumonia-310.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-310.png,Viral Pneumonia/images/Viral Pneumonia-310.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-310.png,Viral Pneumonia/masks/Viral Pneumonia-310.png,True,False,299,299,L,75.37,d2adfdef856261037a8a18f01f493af1
Viral Pneumonia-311.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-311.png,Viral Pneumonia/images/Viral Pneumonia-311.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-311.png,Viral Pneumonia/masks/Viral Pneumonia-311.png,True,False,299,299,L,130.3,eb83122e15e4ee7e2ebc5f12939aae19
Viral Pneumonia-312.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-312.png,Viral Pneumonia/images/Viral Pneumonia-312.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-312.png,Viral Pneumonia/masks/Viral Pneumonia-312.png,True,False,299,299,L,84.1,5e41470d8c264c6ac8916bc5fe6244d1
Viral Pneumonia-313.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-313.png,Viral Pneumonia/images/Viral Pneumonia-313.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-313.png,Viral Pneumonia/masks/Viral Pneumonia-313.png,True,False,299,299,L,106.76,6d4696f69944972106b5a59129798d45
Viral Pneumonia-314.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-314.png,Viral Pneumonia/images/Viral Pneumonia-314.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-314.png,Viral Pneumonia/masks/Viral Pneumonia-314.png,True,False,299,299,L,112.76,668da0e3e76a1e89135f0245e1bac8fc
Viral Pneumonia-315.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-315.png,Viral Pneumonia/images/Viral Pneumonia-315.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-315.png,Viral Pneumonia/masks/Viral Pneumonia-315.png,True,False,299,299,L,109.41,825a79452c95321ed627e38d1a6cf32f
Viral Pneumonia-316.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-316.png,Viral Pneumonia/images/Viral Pneumonia-316.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-316.png,Viral Pneumonia/masks/Viral Pneumonia-316.png,True,False,299,299,L,131.3,b188f7031e6b60721714b054cd02f6e7
Viral Pneumonia-317.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-317.png,Viral Pneumonia/images/Viral Pneumonia-317.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-317.png,Viral Pneumonia/masks/Viral Pneumonia-317.png,True,False,299,299,L,123.44,1a0811409f8d2a3e1320d288ae567218
Viral Pneumonia-318.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-318.png,Viral Pneumonia/images/Viral Pneumonia-318.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-318.png,Viral Pneumonia/masks/Viral Pneumonia-318.png,True,False,299,299,RGB,148.39,191b0a49ec73e0b387d3d21aff7c7cf8
Viral Pneumonia-319.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-319.png,Viral Pneumonia/images/Viral Pneumonia-319.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-319.png,Viral Pneumonia/masks/Viral Pneumonia-319.png,True,False,299,299,RGB,142.47,92adcf8cbb06849605078f41b6e2f03e
Viral Pneumonia-32.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-32.png,Viral Pneumonia/images/Viral Pneumonia-32.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-32.png,Viral Pneumonia/masks/Viral Pneumonia-32.png,True,False,299,299,L,110.52,3b7f081866ba62b89eab013b1d170cb4
Viral Pneumonia-320.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-320.png,Viral Pneumonia/images/Viral Pneumonia-320.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-320.png,Viral Pneumonia/masks/Viral Pneumonia-320.png,True,False,299,299,RGB,139.88,f68b2f500d9dfa8201eba446137df42f
Viral Pneumonia-321.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-321.png,Viral Pneumonia/images/Viral Pneumonia-321.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-321.png,Viral Pneumonia/masks/Viral Pneumonia-321.png,True,False,299,299,L,136.77,bfc4ae9d5ac5e65a94c3c97b2af15734
Viral Pneumonia-322.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-322.png,Viral Pneumonia/images/Viral Pneumonia-322.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-322.png,Viral Pneumonia/masks/Viral Pneumonia-322.png,True,False,299,299,L,150.36,130dbe527cb9794d1872c9b63b0b3c2c
Viral Pneumonia-323.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-323.png,Viral Pneumonia/images/Viral Pneumonia-323.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-323.png,Viral Pneumonia/masks/Viral Pneumonia-323.png,True,False,299,299,RGB,112.08,d271c65ca22f3d24d79fe7b1d8017f3b
Viral Pneumonia-324.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-324.png,Viral Pneumonia/images/Viral Pneumonia-324.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-324.png,Viral Pneumonia/masks/Viral Pneumonia-324.png,True,False,299,299,L,114.46,35c6284a3c8625f47b31aae450ffbf97
Viral Pneumonia-325.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-325.png,Viral Pneumonia/images/Viral Pneumonia-325.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-325.png,Viral Pneumonia/masks/Viral Pneumonia-325.png,True,False,299,299,L,142.42,bbe3c20f0a8bf908a630c77958a73a3e
Viral Pneumonia-326.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-326.png,Viral Pneumonia/images/Viral Pneumonia-326.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-326.png,Viral Pneumonia/masks/Viral Pneumonia-326.png,True,False,299,299,L,118.83,9618988507408219073da012273cfa47
Viral Pneumonia-327.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-327.png,Viral Pneumonia/images/Viral Pneumonia-327.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-327.png,Viral Pneumonia/masks/Viral Pneumonia-327.png,True,False,299,299,RGB,84.71,ca41a38529a86412af616dbc58e355bc
Viral Pneumonia-328.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-328.png,Viral Pneumonia/images/Viral Pneumonia-328.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-328.png,Viral Pneumonia/masks/Viral Pneumonia-328.png,True,False,299,299,L,130.35,a638ea625e3f1b101bd5287dfe1e1073
Viral Pneumonia-329.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-329.png,Viral Pneumonia/images/Viral Pneumonia-329.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-329.png,Viral Pneumonia/masks/Viral Pneumonia-329.png,True,False,299,299,L,125.47,cc728bf0dde57840f0eb68144051e9eb
Viral Pneumonia-33.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-33.png,Viral Pneumonia/images/Viral Pneumonia-33.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-33.png,Viral Pneumonia/masks/Viral Pneumonia-33.png,True,False,299,299,L,134.78,3e1d42002a829631bc5a1d0ee245dedc
Viral Pneumonia-330.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-330.png,Viral Pneumonia/images/Viral Pneumonia-330.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-330.png,Viral Pneumonia/masks/Viral Pneumonia-330.png,True,False,299,299,RGB,123.45,7525e743d8d92b1aecc12a595024e6e1
Viral Pneumonia-331.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-331.png,Viral Pneumonia/images/Viral Pneumonia-331.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-331.png,Viral Pneumonia/masks/Viral Pneumonia-331.png,True,False,299,299,L,121.03,c702a94206cc8319cb97a8d6c2679f8b
Viral Pneumonia-332.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-332.png,Viral Pneumonia/images/Viral Pneumonia-332.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-332.png,Viral Pneumonia/masks/Viral Pneumonia-332.png,True,False,299,299,RGB,152.34,ec175a1a98b3bae4e03a5ea8adebf07b
Viral Pneumonia-333.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-333.png,Viral Pneumonia/images/Viral Pneumonia-333.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-333.png,Viral Pneumonia/masks/Viral Pneumonia-333.png,True,False,299,299,L,118.1,7de723933979b575929ebc7c5b9bb124
Viral Pneumonia-334.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-334.png,Viral Pneumonia/images/Viral Pneumonia-334.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-334.png,Viral Pneumonia/masks/Viral Pneumonia-334.png,True,False,299,299,L,85.49,1473d975873e9bffaa60c878d468affa
Viral Pneumonia-335.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-335.png,Viral Pneumonia/images/Viral Pneumonia-335.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-335.png,Viral Pneumonia/masks/Viral Pneumonia-335.png,True,False,299,299,RGB,142.42,a9a6e16f26c38f269308eb902f5ef6a1
Viral Pneumonia-336.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-336.png,Viral Pneumonia/images/Viral Pneumonia-336.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-336.png,Viral Pneumonia/masks/Viral Pneumonia-336.png,True,False,299,299,L,95.45,d7301873ab6127797bdeae0d7c5e0f89
Viral Pneumonia-337.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-337.png,Viral Pneumonia/images/Viral Pneumonia-337.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-337.png,Viral Pneumonia/masks/Viral Pneumonia-337.png,True,False,299,299,L,103.76,e5587130513e9c64c353e40ecea6abf2
Viral Pneumonia-338.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-338.png,Viral Pneumonia/images/Viral Pneumonia-338.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-338.png,Viral Pneumonia/masks/Viral Pneumonia-338.png,True,False,299,299,L,105.46,05070209e809ca4d82586210343ca92d
Viral Pneumonia-339.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-339.png,Viral Pneumonia/images/Viral Pneumonia-339.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-339.png,Viral Pneumonia/masks/Viral Pneumonia-339.png,True,False,299,299,L,86.07,ffa819e402871326d96657d7b0b89276
Viral Pneumonia-34.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-34.png,Viral Pneumonia/images/Viral Pneumonia-34.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-34.png,Viral Pneumonia/masks/Viral Pneumonia-34.png,True,False,299,299,L,96.33,da85f4d8f3e3f570d185a8d7cb19d5d0
Viral Pneumonia-340.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-340.png,Viral Pneumonia/images/Viral Pneumonia-340.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-340.png,Viral Pneumonia/masks/Viral Pneumonia-340.png,True,False,299,299,L,91.13,820c147e41b1e09946de6201293a9a1f
Viral Pneumonia-341.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-341.png,Viral Pneumonia/images/Viral Pneumonia-341.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-341.png,Viral Pneumonia/masks/Viral Pneumonia-341.png,True,False,299,299,L,120.43,3a555b3a0b0046373d278e131261ca67
Viral Pneumonia-342.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-342.png,Viral Pneumonia/images/Viral Pneumonia-342.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-342.png,Viral Pneumonia/masks/Viral Pneumonia-342.png,True,False,299,299,L,97.83,4ffeac9ef77b8b8ab09b3adf0bd06aad
Viral Pneumonia-343.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-343.png,Viral Pneumonia/images/Viral Pneumonia-343.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-343.png,Viral Pneumonia/masks/Viral Pneumonia-343.png,True,False,299,299,RGB,124.87,70da33946ace4b1762d47b6e1125e452
Viral Pneumonia-344.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-344.png,Viral Pneumonia/images/Viral Pneumonia-344.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-344.png,Viral Pneumonia/masks/Viral Pneumonia-344.png,True,False,299,299,L,122.67,6f2580efc3c95df167b37536d9de45f1
Viral Pneumonia-345.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-345.png,Viral Pneumonia/images/Viral Pneumonia-345.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-345.png,Viral Pneumonia/masks/Viral Pneumonia-345.png,True,False,299,299,L,89.87,18e22f5d056bea201d88b8194982e7c2
Viral Pneumonia-346.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-346.png,Viral Pneumonia/images/Viral Pneumonia-346.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-346.png,Viral Pneumonia/masks/Viral Pneumonia-346.png,True,False,299,299,L,131.72,facd3d2ecca8d5539b40e53ac2b51d6f
Viral Pneumonia-347.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-347.png,Viral Pneumonia/images/Viral Pneumonia-347.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-347.png,Viral Pneumonia/masks/Viral Pneumonia-347.png,True,False,299,299,L,114.74,57a6776943e883b3ff1fbb8b855c979c
Viral Pneumonia-348.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-348.png,Viral Pneumonia/images/Viral Pneumonia-348.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-348.png,Viral Pneumonia/masks/Viral Pneumonia-348.png,True,False,299,299,L,102.55,2ba5868de4a76e2dff6a537c648368e0
Viral Pneumonia-349.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-349.png,Viral Pneumonia/images/Viral Pneumonia-349.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-349.png,Viral Pneumonia/masks/Viral Pneumonia-349.png,True,False,299,299,L,106.99,4597988ee208de0b2f86e23bbf899d7e
Viral Pneumonia-35.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-35.png,Viral Pneumonia/images/Viral Pneumonia-35.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-35.png,Viral Pneumonia/masks/Viral Pneumonia-35.png,True,False,299,299,L,163.08,3ccf86d958a7c7e8bcc0c84b33f97227
Viral Pneumonia-350.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-350.png,Viral Pneumonia/images/Viral Pneumonia-350.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-350.png,Viral Pneumonia/masks/Viral Pneumonia-350.png,True,False,299,299,L,121.71,8c86aaa0071e403cc3ffd6eff2df8426
Viral Pneumonia-351.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-351.png,Viral Pneumonia/images/Viral Pneumonia-351.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-351.png,Viral Pneumonia/masks/Viral Pneumonia-351.png,True,False,299,299,L,84.15,4759a81f49f000e7b4ea5b84559b686d
Viral Pneumonia-352.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-352.png,Viral Pneumonia/images/Viral Pneumonia-352.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-352.png,Viral Pneumonia/masks/Viral Pneumonia-352.png,True,False,299,299,L,119.41,920a8577fdc70f7a7e3a1445c81f641e
Viral Pneumonia-353.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-353.png,Viral Pneumonia/images/Viral Pneumonia-353.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-353.png,Viral Pneumonia/masks/Viral Pneumonia-353.png,True,False,299,299,RGB,119.23,72526b50158a3567b3baac03f17a0e3e
Viral Pneumonia-354.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-354.png,Viral Pneumonia/images/Viral Pneumonia-354.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-354.png,Viral Pneumonia/masks/Viral Pneumonia-354.png,True,False,299,299,L,86.46,982feff7ca5bc517a438bf776c239b6f
Viral Pneumonia-355.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-355.png,Viral Pneumonia/images/Viral Pneumonia-355.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-355.png,Viral Pneumonia/masks/Viral Pneumonia-355.png,True,False,299,299,L,130.46,026d5a9e0402cda0518c51a33a566983
Viral Pneumonia-356.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-356.png,Viral Pneumonia/images/Viral Pneumonia-356.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-356.png,Viral Pneumonia/masks/Viral Pneumonia-356.png,True,False,299,299,L,112.81,cf6ae4db934b0e17f99cdc3c26403540
Viral Pneumonia-357.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-357.png,Viral Pneumonia/images/Viral Pneumonia-357.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-357.png,Viral Pneumonia/masks/Viral Pneumonia-357.png,True,False,299,299,L,140.49,1fb645920e6a82c1d1fac72833958cc0
Viral Pneumonia-358.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-358.png,Viral Pneumonia/images/Viral Pneumonia-358.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-358.png,Viral Pneumonia/masks/Viral Pneumonia-358.png,True,False,299,299,L,144.17,3b0f6d0c2e7c779cc4c1b2160830e96b
Viral Pneumonia-359.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-359.png,Viral Pneumonia/images/Viral Pneumonia-359.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-359.png,Viral Pneumonia/masks/Viral Pneumonia-359.png,True,False,299,299,L,102.03,405b237fd273642f0f28abf6ea4cdada
Viral Pneumonia-36.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-36.png,Viral Pneumonia/images/Viral Pneumonia-36.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-36.png,Viral Pneumonia/masks/Viral Pneumonia-36.png,True,False,299,299,L,144.88,e5ae51265eb33fcea8245427f926a84d
Viral Pneumonia-360.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-360.png,Viral Pneumonia/images/Viral Pneumonia-360.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-360.png,Viral Pneumonia/masks/Viral Pneumonia-360.png,True,False,299,299,L,137.54,ad8f0de9acf613bd0f35bc8061b65666
Viral Pneumonia-361.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-361.png,Viral Pneumonia/images/Viral Pneumonia-361.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-361.png,Viral Pneumonia/masks/Viral Pneumonia-361.png,True,False,299,299,L,113.77,09188615c25223bb3955b1bbf6d0b6ff
Viral Pneumonia-362.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-362.png,Viral Pneumonia/images/Viral Pneumonia-362.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-362.png,Viral Pneumonia/masks/Viral Pneumonia-362.png,True,False,299,299,L,154.16,2d6d0dc67fe60b79b931986f555883b6
Viral Pneumonia-363.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-363.png,Viral Pneumonia/images/Viral Pneumonia-363.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-363.png,Viral Pneumonia/masks/Viral Pneumonia-363.png,True,False,299,299,L,113.5,d7e3fd655ca392dbdaef6c4d6b2e9f40
Viral Pneumonia-364.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-364.png,Viral Pneumonia/images/Viral Pneumonia-364.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-364.png,Viral Pneumonia/masks/Viral Pneumonia-364.png,True,False,299,299,RGB,113.83,f600e8a88b3bb7ff3e9af37798fd9f57
Viral Pneumonia-365.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-365.png,Viral Pneumonia/images/Viral Pneumonia-365.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-365.png,Viral Pneumonia/masks/Viral Pneumonia-365.png,True,False,299,299,L,106.08,58d7955244b49fc0d9e7f2d554cdc7c2
Viral Pneumonia-366.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-366.png,Viral Pneumonia/images/Viral Pneumonia-366.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-366.png,Viral Pneumonia/masks/Viral Pneumonia-366.png,True,False,299,299,L,146.22,79ad727893eb2078f1e153988a738331
Viral Pneumonia-367.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-367.png,Viral Pneumonia/images/Viral Pneumonia-367.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-367.png,Viral Pneumonia/masks/Viral Pneumonia-367.png,True,False,299,299,L,111.92,f3404b143b8da2bd9fcb537a7cb3d566
Viral Pneumonia-368.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-368.png,Viral Pneumonia/images/Viral Pneumonia-368.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-368.png,Viral Pneumonia/masks/Viral Pneumonia-368.png,True,False,299,299,L,116.21,d0deae3f4c8f157876e70060c84512d2
Viral Pneumonia-369.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-369.png,Viral Pneumonia/images/Viral Pneumonia-369.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-369.png,Viral Pneumonia/masks/Viral Pneumonia-369.png,True,False,299,299,L,110.17,361141e052cf4810fccb733102974119
Viral Pneumonia-37.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-37.png,Viral Pneumonia/images/Viral Pneumonia-37.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-37.png,Viral Pneumonia/masks/Viral Pneumonia-37.png,True,False,299,299,L,154.35,0acb304192fc840426305ac5b1eb0749
Viral Pneumonia-370.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-370.png,Viral Pneumonia/images/Viral Pneumonia-370.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-370.png,Viral Pneumonia/masks/Viral Pneumonia-370.png,True,False,299,299,L,101.45,b1fe1a4341d6f2e4e90ea6537b60b1c2
Viral Pneumonia-371.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-371.png,Viral Pneumonia/images/Viral Pneumonia-371.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-371.png,Viral Pneumonia/masks/Viral Pneumonia-371.png,True,False,299,299,L,152.68,f7255eb89aac91013601d4a2aff45621
Viral Pneumonia-372.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-372.png,Viral Pneumonia/images/Viral Pneumonia-372.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-372.png,Viral Pneumonia/masks/Viral Pneumonia-372.png,True,False,299,299,L,105.49,1f2f199433e3dd05a655031137e1cd47
Viral Pneumonia-373.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-373.png,Viral Pneumonia/images/Viral Pneumonia-373.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-373.png,Viral Pneumonia/masks/Viral Pneumonia-373.png,True,False,299,299,RGB,79.52,d98b6c924fb63cb5d83e58044a5e3fe7
Viral Pneumonia-374.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-374.png,Viral Pneumonia/images/Viral Pneumonia-374.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-374.png,Viral Pneumonia/masks/Viral Pneumonia-374.png,True,False,299,299,L,152.29,d566e3904185579e80716217e4044f7e
Viral Pneumonia-375.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-375.png,Viral Pneumonia/images/Viral Pneumonia-375.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-375.png,Viral Pneumonia/masks/Viral Pneumonia-375.png,True,False,299,299,L,160.1,88a9c4311dc5141675428bed3ad5ef02
Viral Pneumonia-376.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-376.png,Viral Pneumonia/images/Viral Pneumonia-376.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-376.png,Viral Pneumonia/masks/Viral Pneumonia-376.png,True,False,299,299,L,99.01,57e587eadd5f59c133d646b96856316c
Viral Pneumonia-377.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-377.png,Viral Pneumonia/images/Viral Pneumonia-377.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-377.png,Viral Pneumonia/masks/Viral Pneumonia-377.png,True,False,299,299,RGB,97.67,e2c45d05f9feb96b5b6799b2fd233bd0
Viral Pneumonia-378.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-378.png,Viral Pneumonia/images/Viral Pneumonia-378.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-378.png,Viral Pneumonia/masks/Viral Pneumonia-378.png,True,False,299,299,L,122.38,665543094590e481198fd431894edb75
Viral Pneumonia-379.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-379.png,Viral Pneumonia/images/Viral Pneumonia-379.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-379.png,Viral Pneumonia/masks/Viral Pneumonia-379.png,True,False,299,299,L,145.67,7387bff89e1b2c4e397290228759a9fa
Viral Pneumonia-38.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-38.png,Viral Pneumonia/images/Viral Pneumonia-38.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-38.png,Viral Pneumonia/masks/Viral Pneumonia-38.png,True,False,299,299,L,131.47,177a3bf3b329f3b8ccfdb117d4f567ba
Viral Pneumonia-380.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-380.png,Viral Pneumonia/images/Viral Pneumonia-380.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-380.png,Viral Pneumonia/masks/Viral Pneumonia-380.png,True,False,299,299,L,110.66,a4851fc32377dfcc8c8ce974a0917fff
Viral Pneumonia-381.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-381.png,Viral Pneumonia/images/Viral Pneumonia-381.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-381.png,Viral Pneumonia/masks/Viral Pneumonia-381.png,True,False,299,299,L,105.34,400fca43a462bdb1bdde7df54a2666c0
Viral Pneumonia-382.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-382.png,Viral Pneumonia/images/Viral Pneumonia-382.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-382.png,Viral Pneumonia/masks/Viral Pneumonia-382.png,True,False,299,299,RGB,143.0,8f2a71558315ca906bff9d83c1d99c94
Viral Pneumonia-383.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-383.png,Viral Pneumonia/images/Viral Pneumonia-383.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-383.png,Viral Pneumonia/masks/Viral Pneumonia-383.png,True,False,299,299,RGB,129.94,448396d9ff7656b46d69da23901f1760
Viral Pneumonia-384.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-384.png,Viral Pneumonia/images/Viral Pneumonia-384.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-384.png,Viral Pneumonia/masks/Viral Pneumonia-384.png,True,False,299,299,L,102.49,793519d89ea71f35a346e8425a96325c
Viral Pneumonia-385.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-385.png,Viral Pneumonia/images/Viral Pneumonia-385.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-385.png,Viral Pneumonia/masks/Viral Pneumonia-385.png,True,False,299,299,RGB,107.75,60ad4fc25115f6552232c437066ba8ac
Viral Pneumonia-386.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-386.png,Viral Pneumonia/images/Viral Pneumonia-386.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-386.png,Viral Pneumonia/masks/Viral Pneumonia-386.png,True,False,299,299,L,110.14,4f37a32c4e50f349cc550be30334544f
Viral Pneumonia-387.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-387.png,Viral Pneumonia/images/Viral Pneumonia-387.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-387.png,Viral Pneumonia/masks/Viral Pneumonia-387.png,True,False,299,299,L,158.1,4338971cd06e9e6a28573fddf30c6cbb
Viral Pneumonia-388.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-388.png,Viral Pneumonia/images/Viral Pneumonia-388.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-388.png,Viral Pneumonia/masks/Viral Pneumonia-388.png,True,False,299,299,RGB,85.95,cf3cf9d727218877839de478f77c4d08
Viral Pneumonia-389.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-389.png,Viral Pneumonia/images/Viral Pneumonia-389.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-389.png,Viral Pneumonia/masks/Viral Pneumonia-389.png,True,False,299,299,RGB,138.81,29e6aebdf493dd4dc6e476c27a2be12a
Viral Pneumonia-39.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-39.png,Viral Pneumonia/images/Viral Pneumonia-39.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-39.png,Viral Pneumonia/masks/Viral Pneumonia-39.png,True,False,299,299,L,173.3,2fc1cb495bd9cf978b6b52852f6b6675
Viral Pneumonia-390.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-390.png,Viral Pneumonia/images/Viral Pneumonia-390.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-390.png,Viral Pneumonia/masks/Viral Pneumonia-390.png,True,False,299,299,RGB,141.11,044fd8546584af1bd0ef037f661c1113
Viral Pneumonia-391.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-391.png,Viral Pneumonia/images/Viral Pneumonia-391.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-391.png,Viral Pneumonia/masks/Viral Pneumonia-391.png,True,False,299,299,RGB,147.98,b00b571147e186b86b1b04966c4fc005
Viral Pneumonia-392.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-392.png,Viral Pneumonia/images/Viral Pneumonia-392.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-392.png,Viral Pneumonia/masks/Viral Pneumonia-392.png,True,False,299,299,L,114.58,e63b563c5f36c0bfa06adaf5825f9ce1
Viral Pneumonia-393.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-393.png,Viral Pneumonia/images/Viral Pneumonia-393.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-393.png,Viral Pneumonia/masks/Viral Pneumonia-393.png,True,False,299,299,L,142.4,176a973db8790140f686a57ee75f6940
Viral Pneumonia-394.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-394.png,Viral Pneumonia/images/Viral Pneumonia-394.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-394.png,Viral Pneumonia/masks/Viral Pneumonia-394.png,True,False,299,299,L,138.92,6519809db686aaf82370f38f7bf302d2
Viral Pneumonia-395.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-395.png,Viral Pneumonia/images/Viral Pneumonia-395.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-395.png,Viral Pneumonia/masks/Viral Pneumonia-395.png,True,False,299,299,L,119.24,6af8ed98bbc56feb4f48bc1840db793c
Viral Pneumonia-396.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-396.png,Viral Pneumonia/images/Viral Pneumonia-396.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-396.png,Viral Pneumonia/masks/Viral Pneumonia-396.png,True,False,299,299,L,121.6,4531aba8f6a9998ff8c3c13de99f408f
Viral Pneumonia-397.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-397.png,Viral Pneumonia/images/Viral Pneumonia-397.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-397.png,Viral Pneumonia/masks/Viral Pneumonia-397.png,True,False,299,299,L,137.92,ce0f073bbd9fd416b9331f0221b865de
Viral Pneumonia-398.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-398.png,Viral Pneumonia/images/Viral Pneumonia-398.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-398.png,Viral Pneumonia/masks/Viral Pneumonia-398.png,True,False,299,299,RGB,139.2,419657e9b13729e4dc6d4a874f1a79a6
Viral Pneumonia-399.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-399.png,Viral Pneumonia/images/Viral Pneumonia-399.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-399.png,Viral Pneumonia/masks/Viral Pneumonia-399.png,True,False,299,299,RGB,157.39,f0fda73281247133b06ba8a39412a84f
Viral Pneumonia-4.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-4.png,Viral Pneumonia/images/Viral Pneumonia-4.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-4.png,Viral Pneumonia/masks/Viral Pneumonia-4.png,True,False,299,299,L,135.64,5a0fad4960b76a95ee0cc4d49d4f3882
Viral Pneumonia-40.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-40.png,Viral Pneumonia/images/Viral Pneumonia-40.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-40.png,Viral Pneumonia/masks/Viral Pneumonia-40.png,True,False,299,299,L,103.47,2eceea8515f8f8703f999305745bc652
Viral Pneumonia-400.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-400.png,Viral Pneumonia/images/Viral Pneumonia-400.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-400.png,Viral Pneumonia/masks/Viral Pneumonia-400.png,True,False,299,299,L,137.26,79b4adc591721b141ee8f9293381a4cd
Viral Pneumonia-401.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-401.png,Viral Pneumonia/images/Viral Pneumonia-401.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-401.png,Viral Pneumonia/masks/Viral Pneumonia-401.png,True,False,299,299,L,126.75,e8ac5195ae4f0666f47c2a6e09d5ea7b
Viral Pneumonia-402.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-402.png,Viral Pneumonia/images/Viral Pneumonia-402.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-402.png,Viral Pneumonia/masks/Viral Pneumonia-402.png,True,False,299,299,L,123.03,10a86ff24ddea4c58aaf2fdb3e082d29
Viral Pneumonia-403.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-403.png,Viral Pneumonia/images/Viral Pneumonia-403.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-403.png,Viral Pneumonia/masks/Viral Pneumonia-403.png,True,False,299,299,L,138.12,d4d59c55636717f74f391e16fea406f6
Viral Pneumonia-404.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-404.png,Viral Pneumonia/images/Viral Pneumonia-404.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-404.png,Viral Pneumonia/masks/Viral Pneumonia-404.png,True,False,299,299,L,112.57,8096b9680618c5480ced145c3c1c1576
Viral Pneumonia-405.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-405.png,Viral Pneumonia/images/Viral Pneumonia-405.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-405.png,Viral Pneumonia/masks/Viral Pneumonia-405.png,True,False,299,299,L,115.88,483d582e221e8c25d180c944fb6ec6ba
Viral Pneumonia-406.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-406.png,Viral Pneumonia/images/Viral Pneumonia-406.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-406.png,Viral Pneumonia/masks/Viral Pneumonia-406.png,True,False,299,299,L,114.53,e6ab9d83fac318c350a94cb368a3d070
Viral Pneumonia-407.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-407.png,Viral Pneumonia/images/Viral Pneumonia-407.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-407.png,Viral Pneumonia/masks/Viral Pneumonia-407.png,True,False,299,299,L,124.6,5a37e923d2f34991a05d911a208d384c
Viral Pneumonia-408.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-408.png,Viral Pneumonia/images/Viral Pneumonia-408.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-408.png,Viral Pneumonia/masks/Viral Pneumonia-408.png,True,False,299,299,RGB,146.92,445684d3b6bd05ebe6fa759d0d174378
Viral Pneumonia-409.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-409.png,Viral Pneumonia/images/Viral Pneumonia-409.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-409.png,Viral Pneumonia/masks/Viral Pneumonia-409.png,True,False,299,299,RGB,154.04,53f072db1c6499d5ff10369cf615344b
Viral Pneumonia-41.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-41.png,Viral Pneumonia/images/Viral Pneumonia-41.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-41.png,Viral Pneumonia/masks/Viral Pneumonia-41.png,True,False,299,299,L,139.08,310e5da8f0f7f1529884e5c572d62bb9
Viral Pneumonia-410.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-410.png,Viral Pneumonia/images/Viral Pneumonia-410.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-410.png,Viral Pneumonia/masks/Viral Pneumonia-410.png,True,False,299,299,L,141.67,9c2cd63573208a6aa209a8c3c1460c59
Viral Pneumonia-411.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-411.png,Viral Pneumonia/images/Viral Pneumonia-411.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-411.png,Viral Pneumonia/masks/Viral Pneumonia-411.png,True,False,299,299,L,106.22,034b55fedc1829b72aef350b3db2af0a
Viral Pneumonia-412.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-412.png,Viral Pneumonia/images/Viral Pneumonia-412.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-412.png,Viral Pneumonia/masks/Viral Pneumonia-412.png,True,False,299,299,L,124.77,963fad6ac820c5763667ee2ca614cbc2
Viral Pneumonia-413.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-413.png,Viral Pneumonia/images/Viral Pneumonia-413.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-413.png,Viral Pneumonia/masks/Viral Pneumonia-413.png,True,False,299,299,L,104.75,7468f194fe983731fb8094c05c6f7fa0
Viral Pneumonia-414.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-414.png,Viral Pneumonia/images/Viral Pneumonia-414.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-414.png,Viral Pneumonia/masks/Viral Pneumonia-414.png,True,False,299,299,L,130.82,4dde1f05243ff3fd1bf5609e36152b95
Viral Pneumonia-415.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-415.png,Viral Pneumonia/images/Viral Pneumonia-415.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-415.png,Viral Pneumonia/masks/Viral Pneumonia-415.png,True,False,299,299,L,117.88,a27a1d34a07c05241f506d6dcbd3f12b
Viral Pneumonia-416.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-416.png,Viral Pneumonia/images/Viral Pneumonia-416.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-416.png,Viral Pneumonia/masks/Viral Pneumonia-416.png,True,False,299,299,L,145.41,cdc33137cffa5d63b8a6b325f0182686
Viral Pneumonia-417.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-417.png,Viral Pneumonia/images/Viral Pneumonia-417.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-417.png,Viral Pneumonia/masks/Viral Pneumonia-417.png,True,False,299,299,L,141.8,46cc967c2d8257f430eeb5a3a18b5f6c
Viral Pneumonia-418.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-418.png,Viral Pneumonia/images/Viral Pneumonia-418.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-418.png,Viral Pneumonia/masks/Viral Pneumonia-418.png,True,False,299,299,L,117.49,62d2253548c22e600685218a193d9364
Viral Pneumonia-419.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-419.png,Viral Pneumonia/images/Viral Pneumonia-419.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-419.png,Viral Pneumonia/masks/Viral Pneumonia-419.png,True,False,299,299,L,129.85,260660d0ace3dcba5832943c09afed1c
Viral Pneumonia-42.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-42.png,Viral Pneumonia/images/Viral Pneumonia-42.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-42.png,Viral Pneumonia/masks/Viral Pneumonia-42.png,True,False,299,299,L,102.09,1d543580fe661c1edeb7f3a25a4723c2
Viral Pneumonia-420.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-420.png,Viral Pneumonia/images/Viral Pneumonia-420.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-420.png,Viral Pneumonia/masks/Viral Pneumonia-420.png,True,False,299,299,L,109.19,b0d9752faa08f017364f039b41bf4052
Viral Pneumonia-421.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-421.png,Viral Pneumonia/images/Viral Pneumonia-421.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-421.png,Viral Pneumonia/masks/Viral Pneumonia-421.png,True,False,299,299,L,119.45,5a75ad6a5d35021e49a6dbe041eed369
Viral Pneumonia-422.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-422.png,Viral Pneumonia/images/Viral Pneumonia-422.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-422.png,Viral Pneumonia/masks/Viral Pneumonia-422.png,True,False,299,299,RGB,163.05,7cbb3b53a5bb4f409fd9b052f0903cde
Viral Pneumonia-423.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-423.png,Viral Pneumonia/images/Viral Pneumonia-423.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-423.png,Viral Pneumonia/masks/Viral Pneumonia-423.png,True,False,299,299,RGB,133.07,ccc6bf667ba1543c52b3f8a4252950c6
Viral Pneumonia-424.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-424.png,Viral Pneumonia/images/Viral Pneumonia-424.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-424.png,Viral Pneumonia/masks/Viral Pneumonia-424.png,True,False,299,299,L,82.07,68230be893070ecba25c88352e4d5466
Viral Pneumonia-425.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-425.png,Viral Pneumonia/images/Viral Pneumonia-425.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-425.png,Viral Pneumonia/masks/Viral Pneumonia-425.png,True,False,299,299,L,118.64,2bb357a863f7b775521f27356d0d9a12
Viral Pneumonia-426.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-426.png,Viral Pneumonia/images/Viral Pneumonia-426.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-426.png,Viral Pneumonia/masks/Viral Pneumonia-426.png,True,False,299,299,L,123.22,72777b7ef10dd3a25b292bd2f07d04f7
Viral Pneumonia-427.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-427.png,Viral Pneumonia/images/Viral Pneumonia-427.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-427.png,Viral Pneumonia/masks/Viral Pneumonia-427.png,True,False,299,299,L,127.45,772d44e596f0bc13c2fc49f2c61e68cc
Viral Pneumonia-428.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-428.png,Viral Pneumonia/images/Viral Pneumonia-428.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-428.png,Viral Pneumonia/masks/Viral Pneumonia-428.png,True,False,299,299,RGB,136.82,bfe29226deda906e42c16ae26920ab86
Viral Pneumonia-429.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-429.png,Viral Pneumonia/images/Viral Pneumonia-429.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-429.png,Viral Pneumonia/masks/Viral Pneumonia-429.png,True,False,299,299,RGB,149.58,651ed4bb79015730b0ad8ab2b2ca30d3
Viral Pneumonia-43.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-43.png,Viral Pneumonia/images/Viral Pneumonia-43.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-43.png,Viral Pneumonia/masks/Viral Pneumonia-43.png,True,False,299,299,L,89.75,248376a9b1a40ad96f1b42a3f09c6ba9
Viral Pneumonia-430.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-430.png,Viral Pneumonia/images/Viral Pneumonia-430.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-430.png,Viral Pneumonia/masks/Viral Pneumonia-430.png,True,False,299,299,RGB,137.76,9efb4a74355f5f3b8e6fd74f907d7e32
Viral Pneumonia-431.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-431.png,Viral Pneumonia/images/Viral Pneumonia-431.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-431.png,Viral Pneumonia/masks/Viral Pneumonia-431.png,True,False,299,299,L,113.8,4501dd55901409ed837486e102f3913a
Viral Pneumonia-432.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-432.png,Viral Pneumonia/images/Viral Pneumonia-432.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-432.png,Viral Pneumonia/masks/Viral Pneumonia-432.png,True,False,299,299,L,122.31,0dc67f3f4f5cc3e36a90cc9f1ba1e88b
Viral Pneumonia-433.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-433.png,Viral Pneumonia/images/Viral Pneumonia-433.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-433.png,Viral Pneumonia/masks/Viral Pneumonia-433.png,True,False,299,299,RGB,134.96,91ea669ec4e058ed24ce76e5f0fec527
Viral Pneumonia-434.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-434.png,Viral Pneumonia/images/Viral Pneumonia-434.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-434.png,Viral Pneumonia/masks/Viral Pneumonia-434.png,True,False,299,299,L,127.88,e7e055e0ad123a82f6f72a7b7e469504
Viral Pneumonia-435.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-435.png,Viral Pneumonia/images/Viral Pneumonia-435.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-435.png,Viral Pneumonia/masks/Viral Pneumonia-435.png,True,False,299,299,L,130.16,83c614cfcf9bca09beb3518e7453e29f
Viral Pneumonia-436.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-436.png,Viral Pneumonia/images/Viral Pneumonia-436.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-436.png,Viral Pneumonia/masks/Viral Pneumonia-436.png,True,False,299,299,RGB,115.23,22c08de9ccddc0486befdd85471610da
Viral Pneumonia-437.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-437.png,Viral Pneumonia/images/Viral Pneumonia-437.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-437.png,Viral Pneumonia/masks/Viral Pneumonia-437.png,True,False,299,299,L,102.0,de9156eb2f58603d44501cac4400b96a
Viral Pneumonia-438.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-438.png,Viral Pneumonia/images/Viral Pneumonia-438.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-438.png,Viral Pneumonia/masks/Viral Pneumonia-438.png,True,False,299,299,L,129.42,e39a12cf7d261910eabdc146116f4d4f
Viral Pneumonia-439.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-439.png,Viral Pneumonia/images/Viral Pneumonia-439.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-439.png,Viral Pneumonia/masks/Viral Pneumonia-439.png,True,False,299,299,L,121.38,ff7c0e8d5c2ed8f8d5db104bd4a67857
Viral Pneumonia-44.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-44.png,Viral Pneumonia/images/Viral Pneumonia-44.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-44.png,Viral Pneumonia/masks/Viral Pneumonia-44.png,True,False,299,299,L,101.19,0edc1c3ee2442358642c9ca899693338
Viral Pneumonia-440.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-440.png,Viral Pneumonia/images/Viral Pneumonia-440.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-440.png,Viral Pneumonia/masks/Viral Pneumonia-440.png,True,False,299,299,L,126.87,344c6b15aab85d9532ee90dd34bb72ad
Viral Pneumonia-441.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-441.png,Viral Pneumonia/images/Viral Pneumonia-441.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-441.png,Viral Pneumonia/masks/Viral Pneumonia-441.png,True,False,299,299,L,134.28,84e94b781ce45bc4f39ff9a8a760b36d
Viral Pneumonia-442.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-442.png,Viral Pneumonia/images/Viral Pneumonia-442.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-442.png,Viral Pneumonia/masks/Viral Pneumonia-442.png,True,False,299,299,L,135.94,e4f507682d3f57ca8118823255ecb6c8
Viral Pneumonia-443.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-443.png,Viral Pneumonia/images/Viral Pneumonia-443.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-443.png,Viral Pneumonia/masks/Viral Pneumonia-443.png,True,False,299,299,L,119.93,d83112010c374916c73d1104bb5b4975
Viral Pneumonia-444.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-444.png,Viral Pneumonia/images/Viral Pneumonia-444.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-444.png,Viral Pneumonia/masks/Viral Pneumonia-444.png,True,False,299,299,L,106.56,233480886a4ce47b45af6f5f9eb41bb0
Viral Pneumonia-445.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-445.png,Viral Pneumonia/images/Viral Pneumonia-445.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-445.png,Viral Pneumonia/masks/Viral Pneumonia-445.png,True,False,299,299,RGB,122.15,f9d818f2e55e1e14e91f85993ac1ec9e
Viral Pneumonia-446.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-446.png,Viral Pneumonia/images/Viral Pneumonia-446.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-446.png,Viral Pneumonia/masks/Viral Pneumonia-446.png,True,False,299,299,L,146.76,9eb31716f5cc684ff56fa0010647bf2b
Viral Pneumonia-447.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-447.png,Viral Pneumonia/images/Viral Pneumonia-447.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-447.png,Viral Pneumonia/masks/Viral Pneumonia-447.png,True,False,299,299,L,129.1,e6af35cf019dc92e3af8e22284e668f9
Viral Pneumonia-448.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-448.png,Viral Pneumonia/images/Viral Pneumonia-448.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-448.png,Viral Pneumonia/masks/Viral Pneumonia-448.png,True,False,299,299,L,117.17,d0d50d6bfbc63687c42a68f2ea05603d
Viral Pneumonia-449.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-449.png,Viral Pneumonia/images/Viral Pneumonia-449.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-449.png,Viral Pneumonia/masks/Viral Pneumonia-449.png,True,False,299,299,L,107.17,b61c95bf4a525b08c99698e0958aa0b2
Viral Pneumonia-45.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-45.png,Viral Pneumonia/images/Viral Pneumonia-45.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-45.png,Viral Pneumonia/masks/Viral Pneumonia-45.png,True,False,299,299,L,134.72,0c8bff80617a9713d32b719bf417a1ba
Viral Pneumonia-450.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-450.png,Viral Pneumonia/images/Viral Pneumonia-450.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-450.png,Viral Pneumonia/masks/Viral Pneumonia-450.png,True,False,299,299,L,106.51,f0bfaa3df47f7ba997d051b6e99a82a5
Viral Pneumonia-451.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-451.png,Viral Pneumonia/images/Viral Pneumonia-451.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-451.png,Viral Pneumonia/masks/Viral Pneumonia-451.png,True,False,299,299,L,102.77,08984e64797ad39847ee2427e4a2888b
Viral Pneumonia-452.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-452.png,Viral Pneumonia/images/Viral Pneumonia-452.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-452.png,Viral Pneumonia/masks/Viral Pneumonia-452.png,True,False,299,299,L,139.3,0c0f07a287d2b28b31505c4edd94acd8
Viral Pneumonia-453.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-453.png,Viral Pneumonia/images/Viral Pneumonia-453.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-453.png,Viral Pneumonia/masks/Viral Pneumonia-453.png,True,False,299,299,L,135.94,145ce6466c4efa121d7a75b07793d829
Viral Pneumonia-454.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-454.png,Viral Pneumonia/images/Viral Pneumonia-454.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-454.png,Viral Pneumonia/masks/Viral Pneumonia-454.png,True,False,299,299,RGB,125.97,9dda4cce0003029512826d6ec29dc507
Viral Pneumonia-455.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-455.png,Viral Pneumonia/images/Viral Pneumonia-455.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-455.png,Viral Pneumonia/masks/Viral Pneumonia-455.png,True,False,299,299,L,99.07,2a442ee05af2b9766593fa7c3997a0ee
Viral Pneumonia-456.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-456.png,Viral Pneumonia/images/Viral Pneumonia-456.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-456.png,Viral Pneumonia/masks/Viral Pneumonia-456.png,True,False,299,299,L,126.74,c43313fc1c6c4a65db04128479bfc5f2
Viral Pneumonia-457.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-457.png,Viral Pneumonia/images/Viral Pneumonia-457.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-457.png,Viral Pneumonia/masks/Viral Pneumonia-457.png,True,False,299,299,L,120.62,83aab89689aa248f9b21f865e01552c4
Viral Pneumonia-458.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-458.png,Viral Pneumonia/images/Viral Pneumonia-458.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-458.png,Viral Pneumonia/masks/Viral Pneumonia-458.png,True,False,299,299,RGB,156.48,cfb94ae031617408513542c9e0b96cab
Viral Pneumonia-459.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-459.png,Viral Pneumonia/images/Viral Pneumonia-459.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-459.png,Viral Pneumonia/masks/Viral Pneumonia-459.png,True,False,299,299,RGB,128.68,de20c14fdb5f08051286af9ee0f2d0be
Viral Pneumonia-46.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-46.png,Viral Pneumonia/images/Viral Pneumonia-46.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-46.png,Viral Pneumonia/masks/Viral Pneumonia-46.png,True,False,299,299,L,154.43,c3bd96cc4f758f389df739de71c55702
Viral Pneumonia-460.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-460.png,Viral Pneumonia/images/Viral Pneumonia-460.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-460.png,Viral Pneumonia/masks/Viral Pneumonia-460.png,True,False,299,299,L,110.87,3c04c83581f6a64ba5bdf7432f9a79a6
Viral Pneumonia-461.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-461.png,Viral Pneumonia/images/Viral Pneumonia-461.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-461.png,Viral Pneumonia/masks/Viral Pneumonia-461.png,True,False,299,299,L,110.18,d6a4c9914872d8624f86d309635838a7
Viral Pneumonia-462.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-462.png,Viral Pneumonia/images/Viral Pneumonia-462.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-462.png,Viral Pneumonia/masks/Viral Pneumonia-462.png,True,False,299,299,L,145.58,b526483bff65bbe8ae6cdc301c329dad
Viral Pneumonia-463.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-463.png,Viral Pneumonia/images/Viral Pneumonia-463.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-463.png,Viral Pneumonia/masks/Viral Pneumonia-463.png,True,False,299,299,L,146.76,1b34a9f4c029d4197873aa6ff416e44d
Viral Pneumonia-464.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-464.png,Viral Pneumonia/images/Viral Pneumonia-464.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-464.png,Viral Pneumonia/masks/Viral Pneumonia-464.png,True,False,299,299,L,124.59,d0c4613ddcdfe40bcbdac796ff4f9469
Viral Pneumonia-465.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-465.png,Viral Pneumonia/images/Viral Pneumonia-465.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-465.png,Viral Pneumonia/masks/Viral Pneumonia-465.png,True,False,299,299,L,128.54,9c3629ffdb5fa4282625e812188635fb
Viral Pneumonia-466.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-466.png,Viral Pneumonia/images/Viral Pneumonia-466.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-466.png,Viral Pneumonia/masks/Viral Pneumonia-466.png,True,False,299,299,L,134.77,6e1e1e696d9b7faa798bfd1408539857
Viral Pneumonia-467.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-467.png,Viral Pneumonia/images/Viral Pneumonia-467.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-467.png,Viral Pneumonia/masks/Viral Pneumonia-467.png,True,False,299,299,L,135.36,41b6a41d3886c62a775ff05653c51b70
Viral Pneumonia-468.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-468.png,Viral Pneumonia/images/Viral Pneumonia-468.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-468.png,Viral Pneumonia/masks/Viral Pneumonia-468.png,True,False,299,299,L,139.97,781b771aa7d4a31829f2f4d456724946
Viral Pneumonia-469.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-469.png,Viral Pneumonia/images/Viral Pneumonia-469.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-469.png,Viral Pneumonia/masks/Viral Pneumonia-469.png,True,False,299,299,L,161.75,d64b3482bc0fd659600d08db6113ea10
Viral Pneumonia-47.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-47.png,Viral Pneumonia/images/Viral Pneumonia-47.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-47.png,Viral Pneumonia/masks/Viral Pneumonia-47.png,True,False,299,299,L,152.1,1b5b48c079e8fb17193b8ad4b2630bb9
Viral Pneumonia-470.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-470.png,Viral Pneumonia/images/Viral Pneumonia-470.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-470.png,Viral Pneumonia/masks/Viral Pneumonia-470.png,True,False,299,299,L,143.98,cfb824bd19e12fb359a34a40ed589d97
Viral Pneumonia-471.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-471.png,Viral Pneumonia/images/Viral Pneumonia-471.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-471.png,Viral Pneumonia/masks/Viral Pneumonia-471.png,True,False,299,299,L,137.31,4414cd46dbe4c5a623c0052289f362a7
Viral Pneumonia-472.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-472.png,Viral Pneumonia/images/Viral Pneumonia-472.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-472.png,Viral Pneumonia/masks/Viral Pneumonia-472.png,True,False,299,299,L,89.13,7b2ac8b97d9623022dab25740c86d9a9
Viral Pneumonia-473.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-473.png,Viral Pneumonia/images/Viral Pneumonia-473.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-473.png,Viral Pneumonia/masks/Viral Pneumonia-473.png,True,False,299,299,RGB,131.57,201e8fcce6d219726e0d421344b9a972
Viral Pneumonia-474.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-474.png,Viral Pneumonia/images/Viral Pneumonia-474.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-474.png,Viral Pneumonia/masks/Viral Pneumonia-474.png,True,False,299,299,L,112.43,9d887f552972bcd825e32ee4015c1789
Viral Pneumonia-475.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-475.png,Viral Pneumonia/images/Viral Pneumonia-475.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-475.png,Viral Pneumonia/masks/Viral Pneumonia-475.png,True,False,299,299,L,136.17,fd45190fc4732f6e9f9c59f5a455c6b9
Viral Pneumonia-476.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-476.png,Viral Pneumonia/images/Viral Pneumonia-476.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-476.png,Viral Pneumonia/masks/Viral Pneumonia-476.png,True,False,299,299,L,117.83,587541e556c6f91e8cea1c6eca964ead
Viral Pneumonia-477.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-477.png,Viral Pneumonia/images/Viral Pneumonia-477.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-477.png,Viral Pneumonia/masks/Viral Pneumonia-477.png,True,False,299,299,L,129.53,3c28f31c4cddf48d66c6e1af848b1275
Viral Pneumonia-478.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-478.png,Viral Pneumonia/images/Viral Pneumonia-478.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-478.png,Viral Pneumonia/masks/Viral Pneumonia-478.png,True,False,299,299,L,144.74,4b95501b1dc279498b07b1fab52bd6d7
Viral Pneumonia-479.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-479.png,Viral Pneumonia/images/Viral Pneumonia-479.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-479.png,Viral Pneumonia/masks/Viral Pneumonia-479.png,True,False,299,299,L,134.47,88068d4865267077f47e188097777816
Viral Pneumonia-48.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-48.png,Viral Pneumonia/images/Viral Pneumonia-48.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-48.png,Viral Pneumonia/masks/Viral Pneumonia-48.png,True,False,299,299,L,104.94,e0597531826ccf298d00c6c817a0f2eb
Viral Pneumonia-480.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-480.png,Viral Pneumonia/images/Viral Pneumonia-480.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-480.png,Viral Pneumonia/masks/Viral Pneumonia-480.png,True,False,299,299,L,124.47,66903491d3c34fe598a8e5547325c69f
Viral Pneumonia-481.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-481.png,Viral Pneumonia/images/Viral Pneumonia-481.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-481.png,Viral Pneumonia/masks/Viral Pneumonia-481.png,True,False,299,299,L,141.19,858ceaabeff054337fc748f9d97cd2e4
Viral Pneumonia-482.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-482.png,Viral Pneumonia/images/Viral Pneumonia-482.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-482.png,Viral Pneumonia/masks/Viral Pneumonia-482.png,True,False,299,299,L,139.3,bb1bb5aced22efa788d81651732a7eec
Viral Pneumonia-483.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-483.png,Viral Pneumonia/images/Viral Pneumonia-483.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-483.png,Viral Pneumonia/masks/Viral Pneumonia-483.png,True,False,299,299,L,146.1,29e24c79daffaaf95a71801eb9dfba5a
Viral Pneumonia-484.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-484.png,Viral Pneumonia/images/Viral Pneumonia-484.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-484.png,Viral Pneumonia/masks/Viral Pneumonia-484.png,True,False,299,299,L,129.15,00bb4a377e827b37481b9a4092e4913f
Viral Pneumonia-485.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-485.png,Viral Pneumonia/images/Viral Pneumonia-485.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-485.png,Viral Pneumonia/masks/Viral Pneumonia-485.png,True,False,299,299,L,140.97,821a134b3adfb347528604448f38bcd3
Viral Pneumonia-486.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-486.png,Viral Pneumonia/images/Viral Pneumonia-486.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-486.png,Viral Pneumonia/masks/Viral Pneumonia-486.png,True,False,299,299,L,138.18,80fa0fe0387d75498d6ee4659a60feac
Viral Pneumonia-487.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-487.png,Viral Pneumonia/images/Viral Pneumonia-487.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-487.png,Viral Pneumonia/masks/Viral Pneumonia-487.png,True,False,299,299,L,117.03,2e71cff7f4b950bc822aea94f826016e
Viral Pneumonia-488.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-488.png,Viral Pneumonia/images/Viral Pneumonia-488.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-488.png,Viral Pneumonia/masks/Viral Pneumonia-488.png,True,False,299,299,L,124.78,9d163697aff958278e72c29700405c27
Viral Pneumonia-489.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-489.png,Viral Pneumonia/images/Viral Pneumonia-489.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-489.png,Viral Pneumonia/masks/Viral Pneumonia-489.png,True,False,299,299,L,135.75,5ac95d15ae147a1ef53b1c2fecb261a2
Viral Pneumonia-49.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-49.png,Viral Pneumonia/images/Viral Pneumonia-49.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-49.png,Viral Pneumonia/masks/Viral Pneumonia-49.png,True,False,299,299,L,156.2,1c61318348fa5fe996e5b5d9e368d150
Viral Pneumonia-490.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-490.png,Viral Pneumonia/images/Viral Pneumonia-490.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-490.png,Viral Pneumonia/masks/Viral Pneumonia-490.png,True,False,299,299,L,118.1,8634571cd1c219c3b88b9b53af9ed941
Viral Pneumonia-491.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-491.png,Viral Pneumonia/images/Viral Pneumonia-491.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-491.png,Viral Pneumonia/masks/Viral Pneumonia-491.png,True,False,299,299,L,108.81,d8421ff8161daa38b45f8df046c26537
Viral Pneumonia-492.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-492.png,Viral Pneumonia/images/Viral Pneumonia-492.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-492.png,Viral Pneumonia/masks/Viral Pneumonia-492.png,True,False,299,299,L,119.94,c47e66f4047095d2049b95de431956f0
Viral Pneumonia-493.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-493.png,Viral Pneumonia/images/Viral Pneumonia-493.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-493.png,Viral Pneumonia/masks/Viral Pneumonia-493.png,True,False,299,299,L,132.02,e57c4ae491281f8df6bacfb393d50cfa
Viral Pneumonia-494.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-494.png,Viral Pneumonia/images/Viral Pneumonia-494.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-494.png,Viral Pneumonia/masks/Viral Pneumonia-494.png,True,False,299,299,L,89.62,a81343d63cacd91e8740e371aab06420
Viral Pneumonia-495.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-495.png,Viral Pneumonia/images/Viral Pneumonia-495.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-495.png,Viral Pneumonia/masks/Viral Pneumonia-495.png,True,False,299,299,RGB,120.43,42b5d5cae7f64ac7f2142b14eba93361
Viral Pneumonia-496.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-496.png,Viral Pneumonia/images/Viral Pneumonia-496.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-496.png,Viral Pneumonia/masks/Viral Pneumonia-496.png,True,False,299,299,L,116.72,8b97276827a76a4fdd209adc94f4e1e5
Viral Pneumonia-497.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-497.png,Viral Pneumonia/images/Viral Pneumonia-497.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-497.png,Viral Pneumonia/masks/Viral Pneumonia-497.png,True,False,299,299,L,126.61,efac0ea86584814ab2b9b6f05798f68f
Viral Pneumonia-498.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-498.png,Viral Pneumonia/images/Viral Pneumonia-498.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-498.png,Viral Pneumonia/masks/Viral Pneumonia-498.png,True,False,299,299,L,129.42,87c7eb0c76ccec01276e3ad4140d563f
Viral Pneumonia-499.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-499.png,Viral Pneumonia/images/Viral Pneumonia-499.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-499.png,Viral Pneumonia/masks/Viral Pneumonia-499.png,True,False,299,299,L,111.26,9ec9852d3b2a168b4ef0b60a53fd4585
Viral Pneumonia-5.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-5.png,Viral Pneumonia/images/Viral Pneumonia-5.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-5.png,Viral Pneumonia/masks/Viral Pneumonia-5.png,True,False,299,299,L,124.2,e5e5f81cef108b5d395e9a01168388fc
Viral Pneumonia-50.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-50.png,Viral Pneumonia/images/Viral Pneumonia-50.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-50.png,Viral Pneumonia/masks/Viral Pneumonia-50.png,True,False,299,299,L,162.59,67b7c2b1135c4b156e5ad1ad2592a09a
Viral Pneumonia-500.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-500.png,Viral Pneumonia/images/Viral Pneumonia-500.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-500.png,Viral Pneumonia/masks/Viral Pneumonia-500.png,True,False,299,299,L,66.24,aca74901f7d349a7a05f7ba77917685d
Viral Pneumonia-501.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-501.png,Viral Pneumonia/images/Viral Pneumonia-501.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-501.png,Viral Pneumonia/masks/Viral Pneumonia-501.png,True,False,299,299,L,102.29,4dd07af66771c23ed03855db6c506302
Viral Pneumonia-502.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-502.png,Viral Pneumonia/images/Viral Pneumonia-502.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-502.png,Viral Pneumonia/masks/Viral Pneumonia-502.png,True,False,299,299,L,142.38,2c01f29f032d73a290f8285048671d54
Viral Pneumonia-503.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-503.png,Viral Pneumonia/images/Viral Pneumonia-503.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-503.png,Viral Pneumonia/masks/Viral Pneumonia-503.png,True,False,299,299,L,146.86,191c5bb3096165f4207cc35cedbbe8dd
Viral Pneumonia-504.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-504.png,Viral Pneumonia/images/Viral Pneumonia-504.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-504.png,Viral Pneumonia/masks/Viral Pneumonia-504.png,True,False,299,299,L,121.36,c92a34ea212d1d3de2b419f8777a8fb7
Viral Pneumonia-505.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-505.png,Viral Pneumonia/images/Viral Pneumonia-505.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-505.png,Viral Pneumonia/masks/Viral Pneumonia-505.png,True,False,299,299,L,117.95,f96453c21435f05e83974819ab8968e2
Viral Pneumonia-506.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-506.png,Viral Pneumonia/images/Viral Pneumonia-506.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-506.png,Viral Pneumonia/masks/Viral Pneumonia-506.png,True,False,299,299,L,132.29,ea53c22fb3c05d07b38f185851a7921a
Viral Pneumonia-507.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-507.png,Viral Pneumonia/images/Viral Pneumonia-507.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-507.png,Viral Pneumonia/masks/Viral Pneumonia-507.png,True,False,299,299,L,124.8,5d1e3cd9a4e35802814dd7c33abee540
Viral Pneumonia-508.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-508.png,Viral Pneumonia/images/Viral Pneumonia-508.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-508.png,Viral Pneumonia/masks/Viral Pneumonia-508.png,True,False,299,299,L,96.8,1a965c579469e997fbd8c5e5ed3c6573
Viral Pneumonia-509.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-509.png,Viral Pneumonia/images/Viral Pneumonia-509.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-509.png,Viral Pneumonia/masks/Viral Pneumonia-509.png,True,False,299,299,L,106.35,93e4b1ba659c1266390111e90068bae8
Viral Pneumonia-51.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-51.png,Viral Pneumonia/images/Viral Pneumonia-51.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-51.png,Viral Pneumonia/masks/Viral Pneumonia-51.png,True,False,299,299,L,151.32,5b71a9b75232e413d80476876293b988
Viral Pneumonia-510.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-510.png,Viral Pneumonia/images/Viral Pneumonia-510.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-510.png,Viral Pneumonia/masks/Viral Pneumonia-510.png,True,False,299,299,L,135.9,7766c27c03549bc592469dbc78dc281d
Viral Pneumonia-511.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-511.png,Viral Pneumonia/images/Viral Pneumonia-511.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-511.png,Viral Pneumonia/masks/Viral Pneumonia-511.png,True,False,299,299,L,134.45,f87ebfc314da21e3d4825b311bb76b39
Viral Pneumonia-512.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-512.png,Viral Pneumonia/images/Viral Pneumonia-512.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-512.png,Viral Pneumonia/masks/Viral Pneumonia-512.png,True,False,299,299,L,145.18,79b8ee1c046e5bfd064cd618b8a25839
Viral Pneumonia-513.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-513.png,Viral Pneumonia/images/Viral Pneumonia-513.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-513.png,Viral Pneumonia/masks/Viral Pneumonia-513.png,True,False,299,299,L,131.79,8a22189817349f1db3da3ea3cf03a53b
Viral Pneumonia-514.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-514.png,Viral Pneumonia/images/Viral Pneumonia-514.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-514.png,Viral Pneumonia/masks/Viral Pneumonia-514.png,True,False,299,299,L,143.37,c27c8ca8569862bcce5bfaf72cac9cbf
Viral Pneumonia-515.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-515.png,Viral Pneumonia/images/Viral Pneumonia-515.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-515.png,Viral Pneumonia/masks/Viral Pneumonia-515.png,True,False,299,299,L,169.84,127aedd87386090ff02cea1532e72496
Viral Pneumonia-516.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-516.png,Viral Pneumonia/images/Viral Pneumonia-516.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-516.png,Viral Pneumonia/masks/Viral Pneumonia-516.png,True,False,299,299,L,159.37,43ded3066801cb77a7a8f8e82efccbad
Viral Pneumonia-517.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-517.png,Viral Pneumonia/images/Viral Pneumonia-517.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-517.png,Viral Pneumonia/masks/Viral Pneumonia-517.png,True,False,299,299,L,135.8,d08326d340bbb15db4d104aed5526e32
Viral Pneumonia-518.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-518.png,Viral Pneumonia/images/Viral Pneumonia-518.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-518.png,Viral Pneumonia/masks/Viral Pneumonia-518.png,True,False,299,299,RGB,116.05,164f95121c7cf2407d973417b045530d
Viral Pneumonia-519.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-519.png,Viral Pneumonia/images/Viral Pneumonia-519.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-519.png,Viral Pneumonia/masks/Viral Pneumonia-519.png,True,False,299,299,RGB,136.75,51e67cdb1e21b0dc3c137718995c785a
Viral Pneumonia-52.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-52.png,Viral Pneumonia/images/Viral Pneumonia-52.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-52.png,Viral Pneumonia/masks/Viral Pneumonia-52.png,True,False,299,299,L,140.76,20e14ac6395387a17c5f9e11280a5538
Viral Pneumonia-520.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-520.png,Viral Pneumonia/images/Viral Pneumonia-520.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-520.png,Viral Pneumonia/masks/Viral Pneumonia-520.png,True,False,299,299,RGB,115.05,309119b357719019f9314a93b6f0eb61
Viral Pneumonia-521.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-521.png,Viral Pneumonia/images/Viral Pneumonia-521.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-521.png,Viral Pneumonia/masks/Viral Pneumonia-521.png,True,False,299,299,L,135.95,89a8c63a83d60694467e301e45087a62
Viral Pneumonia-522.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-522.png,Viral Pneumonia/images/Viral Pneumonia-522.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-522.png,Viral Pneumonia/masks/Viral Pneumonia-522.png,True,False,299,299,RGB,164.86,25cdc98f2254bcd6cdbd40fdb0461b28
Viral Pneumonia-523.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-523.png,Viral Pneumonia/images/Viral Pneumonia-523.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-523.png,Viral Pneumonia/masks/Viral Pneumonia-523.png,True,False,299,299,L,132.4,dc45fdf4b494761e84d0267f7b270027
Viral Pneumonia-524.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-524.png,Viral Pneumonia/images/Viral Pneumonia-524.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-524.png,Viral Pneumonia/masks/Viral Pneumonia-524.png,True,False,299,299,L,127.68,c5e97643781b2a3f0c5fca0771b8f094
Viral Pneumonia-525.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-525.png,Viral Pneumonia/images/Viral Pneumonia-525.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-525.png,Viral Pneumonia/masks/Viral Pneumonia-525.png,True,False,299,299,L,122.74,803cc18a711e2dabf29c06b0c9fe508e
Viral Pneumonia-526.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-526.png,Viral Pneumonia/images/Viral Pneumonia-526.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-526.png,Viral Pneumonia/masks/Viral Pneumonia-526.png,True,False,299,299,L,82.12,0427f4f7d1646e510e67dc4e5be2eb11
Viral Pneumonia-527.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-527.png,Viral Pneumonia/images/Viral Pneumonia-527.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-527.png,Viral Pneumonia/masks/Viral Pneumonia-527.png,True,False,299,299,L,140.85,1bd22b96df565e716e08bc1a6a530581
Viral Pneumonia-528.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-528.png,Viral Pneumonia/images/Viral Pneumonia-528.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-528.png,Viral Pneumonia/masks/Viral Pneumonia-528.png,True,False,299,299,L,127.66,99e3b7690ce3953b43e6a3a324b259da
Viral Pneumonia-529.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-529.png,Viral Pneumonia/images/Viral Pneumonia-529.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-529.png,Viral Pneumonia/masks/Viral Pneumonia-529.png,True,False,299,299,L,115.8,71d1f20553de8d2a6dca3e4fc6480273
Viral Pneumonia-53.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-53.png,Viral Pneumonia/images/Viral Pneumonia-53.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-53.png,Viral Pneumonia/masks/Viral Pneumonia-53.png,True,False,299,299,L,126.69,6b4d8d1708ada1e954f5cecbc5796e19
Viral Pneumonia-530.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-530.png,Viral Pneumonia/images/Viral Pneumonia-530.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-530.png,Viral Pneumonia/masks/Viral Pneumonia-530.png,True,False,299,299,L,139.23,35bf7a0ceab8f4ccca210eca5404a4cf
Viral Pneumonia-531.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-531.png,Viral Pneumonia/images/Viral Pneumonia-531.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-531.png,Viral Pneumonia/masks/Viral Pneumonia-531.png,True,False,299,299,L,145.67,a99d97895af8ccbfdd9adcf2be904519
Viral Pneumonia-532.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-532.png,Viral Pneumonia/images/Viral Pneumonia-532.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-532.png,Viral Pneumonia/masks/Viral Pneumonia-532.png,True,False,299,299,L,135.11,00eac1dd52daf8c325e7cb6fdcac69e9
Viral Pneumonia-533.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-533.png,Viral Pneumonia/images/Viral Pneumonia-533.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-533.png,Viral Pneumonia/masks/Viral Pneumonia-533.png,True,False,299,299,L,115.53,46dabc7dc70b9c71c273647e5a1e5eec
Viral Pneumonia-534.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-534.png,Viral Pneumonia/images/Viral Pneumonia-534.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-534.png,Viral Pneumonia/masks/Viral Pneumonia-534.png,True,False,299,299,L,149.91,845a683d985b1f61461f1b2946da0588
Viral Pneumonia-535.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-535.png,Viral Pneumonia/images/Viral Pneumonia-535.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-535.png,Viral Pneumonia/masks/Viral Pneumonia-535.png,True,False,299,299,L,101.11,adeb048c1db171385f25979cea16ca77
Viral Pneumonia-536.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-536.png,Viral Pneumonia/images/Viral Pneumonia-536.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-536.png,Viral Pneumonia/masks/Viral Pneumonia-536.png,True,False,299,299,L,117.41,1797ac82ecc400a0559eaeae3c76af26
Viral Pneumonia-537.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-537.png,Viral Pneumonia/images/Viral Pneumonia-537.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-537.png,Viral Pneumonia/masks/Viral Pneumonia-537.png,True,False,299,299,L,128.42,8e90402076cd8b7a65d84de3876e0db7
Viral Pneumonia-538.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-538.png,Viral Pneumonia/images/Viral Pneumonia-538.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-538.png,Viral Pneumonia/masks/Viral Pneumonia-538.png,True,False,299,299,L,129.68,8becca511008505da878f32f4ab118a0
Viral Pneumonia-539.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-539.png,Viral Pneumonia/images/Viral Pneumonia-539.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-539.png,Viral Pneumonia/masks/Viral Pneumonia-539.png,True,False,299,299,L,138.43,1ff06fe966fca03293958b069b513c92
Viral Pneumonia-54.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-54.png,Viral Pneumonia/images/Viral Pneumonia-54.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-54.png,Viral Pneumonia/masks/Viral Pneumonia-54.png,True,False,299,299,L,154.93,66dd48ea2a1a8a97f751068205cc0aa4
Viral Pneumonia-540.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-540.png,Viral Pneumonia/images/Viral Pneumonia-540.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-540.png,Viral Pneumonia/masks/Viral Pneumonia-540.png,True,False,299,299,L,138.15,1ea24af2bf3bf4a9ef15bb95dfb46da3
Viral Pneumonia-541.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-541.png,Viral Pneumonia/images/Viral Pneumonia-541.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-541.png,Viral Pneumonia/masks/Viral Pneumonia-541.png,True,False,299,299,RGB,140.3,e894bb807d18a7ced5c8b0fd5de8dd94
Viral Pneumonia-542.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-542.png,Viral Pneumonia/images/Viral Pneumonia-542.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-542.png,Viral Pneumonia/masks/Viral Pneumonia-542.png,True,False,299,299,L,127.61,17a7588a54e6c6e36006ab53a242a0c0
Viral Pneumonia-543.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-543.png,Viral Pneumonia/images/Viral Pneumonia-543.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-543.png,Viral Pneumonia/masks/Viral Pneumonia-543.png,True,False,299,299,L,149.29,08cda50c0d7a7649637ceec9c22a32d6
Viral Pneumonia-544.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-544.png,Viral Pneumonia/images/Viral Pneumonia-544.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-544.png,Viral Pneumonia/masks/Viral Pneumonia-544.png,True,False,299,299,L,137.18,29d927146f254b74aa4432b456542338
Viral Pneumonia-545.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-545.png,Viral Pneumonia/images/Viral Pneumonia-545.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-545.png,Viral Pneumonia/masks/Viral Pneumonia-545.png,True,False,299,299,L,138.22,f24170ed96727bf6dcb7aeb5402eba44
Viral Pneumonia-546.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-546.png,Viral Pneumonia/images/Viral Pneumonia-546.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-546.png,Viral Pneumonia/masks/Viral Pneumonia-546.png,True,False,299,299,L,137.74,992482115bb824dbaad46019e7ab0336
Viral Pneumonia-547.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-547.png,Viral Pneumonia/images/Viral Pneumonia-547.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-547.png,Viral Pneumonia/masks/Viral Pneumonia-547.png,True,False,299,299,L,165.31,0a4ffde0f8437a42a5932a666b909f5f
Viral Pneumonia-548.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-548.png,Viral Pneumonia/images/Viral Pneumonia-548.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-548.png,Viral Pneumonia/masks/Viral Pneumonia-548.png,True,False,299,299,L,105.51,8332aa91e47bc8d15047556dd2c1a6bd
Viral Pneumonia-549.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-549.png,Viral Pneumonia/images/Viral Pneumonia-549.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-549.png,Viral Pneumonia/masks/Viral Pneumonia-549.png,True,False,299,299,L,136.42,f5e6df474a153dca3880f0eb72d1113b
Viral Pneumonia-55.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-55.png,Viral Pneumonia/images/Viral Pneumonia-55.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-55.png,Viral Pneumonia/masks/Viral Pneumonia-55.png,True,False,299,299,L,128.18,0271798419deb23b47d5243e1e44c572
Viral Pneumonia-550.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-550.png,Viral Pneumonia/images/Viral Pneumonia-550.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-550.png,Viral Pneumonia/masks/Viral Pneumonia-550.png,True,False,299,299,L,127.31,8432be79c284041acca5debf417afca9
Viral Pneumonia-551.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-551.png,Viral Pneumonia/images/Viral Pneumonia-551.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-551.png,Viral Pneumonia/masks/Viral Pneumonia-551.png,True,False,299,299,RGB,139.82,67ece0f7b997d586a2708dcf7850edc5
Viral Pneumonia-552.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-552.png,Viral Pneumonia/images/Viral Pneumonia-552.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-552.png,Viral Pneumonia/masks/Viral Pneumonia-552.png,True,False,299,299,L,135.49,f50a326d3b484f2c8f59efd1357a8c42
Viral Pneumonia-553.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-553.png,Viral Pneumonia/images/Viral Pneumonia-553.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-553.png,Viral Pneumonia/masks/Viral Pneumonia-553.png,True,False,299,299,L,131.78,a7655da2d642629fc387fb100151936b
Viral Pneumonia-554.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-554.png,Viral Pneumonia/images/Viral Pneumonia-554.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-554.png,Viral Pneumonia/masks/Viral Pneumonia-554.png,True,False,299,299,L,123.2,1541e18cafb0558bade41f097aaee1bf
Viral Pneumonia-555.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-555.png,Viral Pneumonia/images/Viral Pneumonia-555.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-555.png,Viral Pneumonia/masks/Viral Pneumonia-555.png,True,False,299,299,L,146.09,b43618f70aec122ed255c7bea6035293
Viral Pneumonia-556.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-556.png,Viral Pneumonia/images/Viral Pneumonia-556.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-556.png,Viral Pneumonia/masks/Viral Pneumonia-556.png,True,False,299,299,L,93.53,92b504abfaeeb7d34e950b9a33515f9d
Viral Pneumonia-557.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-557.png,Viral Pneumonia/images/Viral Pneumonia-557.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-557.png,Viral Pneumonia/masks/Viral Pneumonia-557.png,True,False,299,299,L,125.53,f98b708547a7f31510594b59e998da57
Viral Pneumonia-558.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-558.png,Viral Pneumonia/images/Viral Pneumonia-558.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-558.png,Viral Pneumonia/masks/Viral Pneumonia-558.png,True,False,299,299,L,95.94,232711dd0e0059ff3d7e482983a4e053
Viral Pneumonia-559.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-559.png,Viral Pneumonia/images/Viral Pneumonia-559.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-559.png,Viral Pneumonia/masks/Viral Pneumonia-559.png,True,False,299,299,L,150.97,9aaf8694b0640758797147ae8c9100c3
Viral Pneumonia-56.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-56.png,Viral Pneumonia/images/Viral Pneumonia-56.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-56.png,Viral Pneumonia/masks/Viral Pneumonia-56.png,True,False,299,299,L,130.59,761a9ad7fe02bbe54b83382d44ec0323
Viral Pneumonia-560.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-560.png,Viral Pneumonia/images/Viral Pneumonia-560.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-560.png,Viral Pneumonia/masks/Viral Pneumonia-560.png,True,False,299,299,L,128.71,1bf76e1db5fb29bc0e58a798ade90d34
Viral Pneumonia-561.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-561.png,Viral Pneumonia/images/Viral Pneumonia-561.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-561.png,Viral Pneumonia/masks/Viral Pneumonia-561.png,True,False,299,299,L,121.84,4c2160447648d4cd66118b854921b2b0
Viral Pneumonia-562.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-562.png,Viral Pneumonia/images/Viral Pneumonia-562.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-562.png,Viral Pneumonia/masks/Viral Pneumonia-562.png,True,False,299,299,L,121.23,085b94b8ad7a23f757292bb636b4a1f8
Viral Pneumonia-563.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-563.png,Viral Pneumonia/images/Viral Pneumonia-563.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-563.png,Viral Pneumonia/masks/Viral Pneumonia-563.png,True,False,299,299,L,164.74,8b93273d0e4f1558fd13d4d0a7efa956
Viral Pneumonia-564.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-564.png,Viral Pneumonia/images/Viral Pneumonia-564.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-564.png,Viral Pneumonia/masks/Viral Pneumonia-564.png,True,False,299,299,L,113.22,c9500af62448b0ec8461e9848c7b0639
Viral Pneumonia-565.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-565.png,Viral Pneumonia/images/Viral Pneumonia-565.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-565.png,Viral Pneumonia/masks/Viral Pneumonia-565.png,True,False,299,299,L,126.86,ef1579ab0df05945d1715b39db3ead84
Viral Pneumonia-566.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-566.png,Viral Pneumonia/images/Viral Pneumonia-566.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-566.png,Viral Pneumonia/masks/Viral Pneumonia-566.png,True,False,299,299,L,149.57,13c31bc8a68e7c6e3a4ee597dc90e55c
Viral Pneumonia-567.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-567.png,Viral Pneumonia/images/Viral Pneumonia-567.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-567.png,Viral Pneumonia/masks/Viral Pneumonia-567.png,True,False,299,299,L,108.7,132aaaaf3fa52da91b1702572fa90d39
Viral Pneumonia-568.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-568.png,Viral Pneumonia/images/Viral Pneumonia-568.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-568.png,Viral Pneumonia/masks/Viral Pneumonia-568.png,True,False,299,299,L,106.27,f2c6505648ea155b23ea057d89ad037a
Viral Pneumonia-569.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-569.png,Viral Pneumonia/images/Viral Pneumonia-569.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-569.png,Viral Pneumonia/masks/Viral Pneumonia-569.png,True,False,299,299,L,151.93,2d42aed489c81bb7d356563e695b6fbd
Viral Pneumonia-57.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-57.png,Viral Pneumonia/images/Viral Pneumonia-57.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-57.png,Viral Pneumonia/masks/Viral Pneumonia-57.png,True,False,299,299,L,117.73,333ea57260998ac01100d774c7c0ae37
Viral Pneumonia-570.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-570.png,Viral Pneumonia/images/Viral Pneumonia-570.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-570.png,Viral Pneumonia/masks/Viral Pneumonia-570.png,True,False,299,299,L,145.28,6cae32a120cab35d7cba576e2a7cf97f
Viral Pneumonia-571.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-571.png,Viral Pneumonia/images/Viral Pneumonia-571.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-571.png,Viral Pneumonia/masks/Viral Pneumonia-571.png,True,False,299,299,L,119.25,01201797c440132b21afdd458cbbc961
Viral Pneumonia-572.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-572.png,Viral Pneumonia/images/Viral Pneumonia-572.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-572.png,Viral Pneumonia/masks/Viral Pneumonia-572.png,True,False,299,299,L,122.79,27da8dc3e55d688c7cfb7d2dfa836bbe
Viral Pneumonia-573.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-573.png,Viral Pneumonia/images/Viral Pneumonia-573.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-573.png,Viral Pneumonia/masks/Viral Pneumonia-573.png,True,False,299,299,L,131.6,2e0604eaaae27c5ff95da2ffefdec37e
Viral Pneumonia-574.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-574.png,Viral Pneumonia/images/Viral Pneumonia-574.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-574.png,Viral Pneumonia/masks/Viral Pneumonia-574.png,True,False,299,299,L,127.62,c4b6984dda26e291ff10a3009ecf24bc
Viral Pneumonia-575.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-575.png,Viral Pneumonia/images/Viral Pneumonia-575.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-575.png,Viral Pneumonia/masks/Viral Pneumonia-575.png,True,False,299,299,L,144.8,b6fe0557503aeeeae9dd8311afbc472c
Viral Pneumonia-576.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-576.png,Viral Pneumonia/images/Viral Pneumonia-576.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-576.png,Viral Pneumonia/masks/Viral Pneumonia-576.png,True,False,299,299,L,139.95,ac1a22b9ed144cdd2a44c5d1339aeab5
Viral Pneumonia-577.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-577.png,Viral Pneumonia/images/Viral Pneumonia-577.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-577.png,Viral Pneumonia/masks/Viral Pneumonia-577.png,True,False,299,299,RGB,126.29,196d83b73fd32978b6543a0c77aa4a25
Viral Pneumonia-578.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-578.png,Viral Pneumonia/images/Viral Pneumonia-578.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-578.png,Viral Pneumonia/masks/Viral Pneumonia-578.png,True,False,299,299,L,109.13,25e16b64cdc8ef7dab0174e2ce0b9e1c
Viral Pneumonia-579.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-579.png,Viral Pneumonia/images/Viral Pneumonia-579.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-579.png,Viral Pneumonia/masks/Viral Pneumonia-579.png,True,False,299,299,L,142.46,6cfb6779de86d5241d224c5edd2be90a
Viral Pneumonia-58.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-58.png,Viral Pneumonia/images/Viral Pneumonia-58.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-58.png,Viral Pneumonia/masks/Viral Pneumonia-58.png,True,False,299,299,L,127.11,6dd9e1f3ee849fe2a49adab5cea81782
Viral Pneumonia-580.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-580.png,Viral Pneumonia/images/Viral Pneumonia-580.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-580.png,Viral Pneumonia/masks/Viral Pneumonia-580.png,True,False,299,299,L,96.1,7d19405c852bb0e3a95d8ad13529406c
Viral Pneumonia-581.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-581.png,Viral Pneumonia/images/Viral Pneumonia-581.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-581.png,Viral Pneumonia/masks/Viral Pneumonia-581.png,True,False,299,299,RGB,143.23,5482a00a29355d943d1a096409cb01c7
Viral Pneumonia-582.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-582.png,Viral Pneumonia/images/Viral Pneumonia-582.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-582.png,Viral Pneumonia/masks/Viral Pneumonia-582.png,True,False,299,299,L,134.0,aa2b320d1d245f81900b3295006b2edc
Viral Pneumonia-583.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-583.png,Viral Pneumonia/images/Viral Pneumonia-583.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-583.png,Viral Pneumonia/masks/Viral Pneumonia-583.png,True,False,299,299,L,141.12,bafedb63b8d2610297031de5e408ede9
Viral Pneumonia-584.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-584.png,Viral Pneumonia/images/Viral Pneumonia-584.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-584.png,Viral Pneumonia/masks/Viral Pneumonia-584.png,True,False,299,299,RGB,144.76,ddcbf873149f4fb56550109b87fe6f65
Viral Pneumonia-585.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-585.png,Viral Pneumonia/images/Viral Pneumonia-585.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-585.png,Viral Pneumonia/masks/Viral Pneumonia-585.png,True,False,299,299,RGB,108.56,c7d19dab8c51c68169321767d652251c
Viral Pneumonia-586.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-586.png,Viral Pneumonia/images/Viral Pneumonia-586.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-586.png,Viral Pneumonia/masks/Viral Pneumonia-586.png,True,False,299,299,RGB,117.2,f2b17c6101a902e8e21575b6e41c5240
Viral Pneumonia-587.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-587.png,Viral Pneumonia/images/Viral Pneumonia-587.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-587.png,Viral Pneumonia/masks/Viral Pneumonia-587.png,True,False,299,299,RGB,143.88,370b67776f9ee4ecb01e9be5ebb74f6b
Viral Pneumonia-588.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-588.png,Viral Pneumonia/images/Viral Pneumonia-588.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-588.png,Viral Pneumonia/masks/Viral Pneumonia-588.png,True,False,299,299,L,129.07,a0ae775b622f03a00f9f7c5f0b1e800e
Viral Pneumonia-589.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-589.png,Viral Pneumonia/images/Viral Pneumonia-589.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-589.png,Viral Pneumonia/masks/Viral Pneumonia-589.png,True,False,299,299,L,133.91,4b29a2d7380a64776d07849a73ae719d
Viral Pneumonia-59.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-59.png,Viral Pneumonia/images/Viral Pneumonia-59.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-59.png,Viral Pneumonia/masks/Viral Pneumonia-59.png,True,False,299,299,L,136.04,119e14d9e376223851ec0bb1ab53e507
Viral Pneumonia-590.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-590.png,Viral Pneumonia/images/Viral Pneumonia-590.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-590.png,Viral Pneumonia/masks/Viral Pneumonia-590.png,True,False,299,299,L,84.63,2f5bf845d1a8e49cf9e16cd8e133c0c3
Viral Pneumonia-591.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-591.png,Viral Pneumonia/images/Viral Pneumonia-591.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-591.png,Viral Pneumonia/masks/Viral Pneumonia-591.png,True,False,299,299,L,142.94,cd1a9fca25d625179d41ca0887c60449
Viral Pneumonia-592.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-592.png,Viral Pneumonia/images/Viral Pneumonia-592.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-592.png,Viral Pneumonia/masks/Viral Pneumonia-592.png,True,False,299,299,L,109.32,79f6712256fe8a71a42c08351b77bb71
Viral Pneumonia-593.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-593.png,Viral Pneumonia/images/Viral Pneumonia-593.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-593.png,Viral Pneumonia/masks/Viral Pneumonia-593.png,True,False,299,299,L,107.41,f9911819293ffb862e285ef2bebe29c4
Viral Pneumonia-594.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-594.png,Viral Pneumonia/images/Viral Pneumonia-594.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-594.png,Viral Pneumonia/masks/Viral Pneumonia-594.png,True,False,299,299,L,133.53,dfb31039c18e1610a3ef2bc70619675c
Viral Pneumonia-595.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-595.png,Viral Pneumonia/images/Viral Pneumonia-595.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-595.png,Viral Pneumonia/masks/Viral Pneumonia-595.png,True,False,299,299,L,129.0,770a1ed23df623f771fdecc5c06b464f
Viral Pneumonia-596.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-596.png,Viral Pneumonia/images/Viral Pneumonia-596.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-596.png,Viral Pneumonia/masks/Viral Pneumonia-596.png,True,False,299,299,L,129.0,770a1ed23df623f771fdecc5c06b464f
Viral Pneumonia-597.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-597.png,Viral Pneumonia/images/Viral Pneumonia-597.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-597.png,Viral Pneumonia/masks/Viral Pneumonia-597.png,True,False,299,299,L,85.12,0210e69328e25a002eba27e9081588a8
Viral Pneumonia-598.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-598.png,Viral Pneumonia/images/Viral Pneumonia-598.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-598.png,Viral Pneumonia/masks/Viral Pneumonia-598.png,True,False,299,299,L,146.63,969cebebee932de25e083c4c31401cbe
Viral Pneumonia-599.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-599.png,Viral Pneumonia/images/Viral Pneumonia-599.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-599.png,Viral Pneumonia/masks/Viral Pneumonia-599.png,True,False,299,299,L,107.55,8599c01f40bd86439695a0ebf55676d4
Viral Pneumonia-6.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-6.png,Viral Pneumonia/images/Viral Pneumonia-6.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-6.png,Viral Pneumonia/masks/Viral Pneumonia-6.png,True,False,299,299,L,113.5,f9f020577043752933b622c68c2aa48a
Viral Pneumonia-60.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-60.png,Viral Pneumonia/images/Viral Pneumonia-60.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-60.png,Viral Pneumonia/masks/Viral Pneumonia-60.png,True,False,299,299,L,146.86,9ed10473ff5424eb6a31b7d1a805acc0
Viral Pneumonia-600.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-600.png,Viral Pneumonia/images/Viral Pneumonia-600.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-600.png,Viral Pneumonia/masks/Viral Pneumonia-600.png,True,False,299,299,L,111.04,2709e39b0dbfaf5b271a2a1c7e431297
Viral Pneumonia-601.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-601.png,Viral Pneumonia/images/Viral Pneumonia-601.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-601.png,Viral Pneumonia/masks/Viral Pneumonia-601.png,True,False,299,299,L,115.7,0d0256131f3b726c516d8409285e1e09
Viral Pneumonia-602.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-602.png,Viral Pneumonia/images/Viral Pneumonia-602.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-602.png,Viral Pneumonia/masks/Viral Pneumonia-602.png,True,False,299,299,L,111.57,76864ede30a0a30b25a32a88eaba2466
Viral Pneumonia-603.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-603.png,Viral Pneumonia/images/Viral Pneumonia-603.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-603.png,Viral Pneumonia/masks/Viral Pneumonia-603.png,True,False,299,299,L,117.42,835977cccb07c2e23acb5ddea58ce2c1
Viral Pneumonia-604.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-604.png,Viral Pneumonia/images/Viral Pneumonia-604.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-604.png,Viral Pneumonia/masks/Viral Pneumonia-604.png,True,False,299,299,L,120.23,aebd9cab4c7c558bc50eb463756a2df5
Viral Pneumonia-605.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-605.png,Viral Pneumonia/images/Viral Pneumonia-605.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-605.png,Viral Pneumonia/masks/Viral Pneumonia-605.png,True,False,299,299,L,135.63,c2d608467d08bcfe675f88e6e62bec72
Viral Pneumonia-606.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-606.png,Viral Pneumonia/images/Viral Pneumonia-606.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-606.png,Viral Pneumonia/masks/Viral Pneumonia-606.png,True,False,299,299,L,104.54,7e8ec38d32ca3ee3871baafa515f56de
Viral Pneumonia-607.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-607.png,Viral Pneumonia/images/Viral Pneumonia-607.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-607.png,Viral Pneumonia/masks/Viral Pneumonia-607.png,True,False,299,299,L,101.64,18b64f2ca79a699cc9ba49de3757d874
Viral Pneumonia-608.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-608.png,Viral Pneumonia/images/Viral Pneumonia-608.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-608.png,Viral Pneumonia/masks/Viral Pneumonia-608.png,True,False,299,299,L,116.15,7c4c24abac5e40d583b1e125167e44a2
Viral Pneumonia-609.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-609.png,Viral Pneumonia/images/Viral Pneumonia-609.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-609.png,Viral Pneumonia/masks/Viral Pneumonia-609.png,True,False,299,299,L,106.61,dc6d82395587a1f5bb022a51fa90d332
Viral Pneumonia-61.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-61.png,Viral Pneumonia/images/Viral Pneumonia-61.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-61.png,Viral Pneumonia/masks/Viral Pneumonia-61.png,True,False,299,299,L,125.16,5e3b58aa92fc83aef6a67150c0c5121a
Viral Pneumonia-610.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-610.png,Viral Pneumonia/images/Viral Pneumonia-610.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-610.png,Viral Pneumonia/masks/Viral Pneumonia-610.png,True,False,299,299,RGB,147.89,caa66164337a7ec6c89e1b5e91e8fafe
Viral Pneumonia-611.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-611.png,Viral Pneumonia/images/Viral Pneumonia-611.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-611.png,Viral Pneumonia/masks/Viral Pneumonia-611.png,True,False,299,299,L,82.08,357401589a193610ec23cd659d75427b
Viral Pneumonia-612.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-612.png,Viral Pneumonia/images/Viral Pneumonia-612.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-612.png,Viral Pneumonia/masks/Viral Pneumonia-612.png,True,False,299,299,RGB,130.59,7b000dfa5b8fed6f3e40bdfa1f4ca5f9
Viral Pneumonia-613.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-613.png,Viral Pneumonia/images/Viral Pneumonia-613.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-613.png,Viral Pneumonia/masks/Viral Pneumonia-613.png,True,False,299,299,RGB,115.88,4929ece3e65cd47e50efd139897d0b32
Viral Pneumonia-614.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-614.png,Viral Pneumonia/images/Viral Pneumonia-614.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-614.png,Viral Pneumonia/masks/Viral Pneumonia-614.png,True,False,299,299,L,108.87,dc6dd1d09041bc16bc1ccbfbee3f0bc5
Viral Pneumonia-615.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-615.png,Viral Pneumonia/images/Viral Pneumonia-615.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-615.png,Viral Pneumonia/masks/Viral Pneumonia-615.png,True,False,299,299,L,83.74,00c4ca3ea87add7d09677e2a14e9e059
Viral Pneumonia-616.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-616.png,Viral Pneumonia/images/Viral Pneumonia-616.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-616.png,Viral Pneumonia/masks/Viral Pneumonia-616.png,True,False,299,299,L,103.26,b8a6f0d7b157a69fe9bf72a186afde2a
Viral Pneumonia-617.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-617.png,Viral Pneumonia/images/Viral Pneumonia-617.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-617.png,Viral Pneumonia/masks/Viral Pneumonia-617.png,True,False,299,299,L,95.16,1019856e97299c5657d2c75d6054cc6a
Viral Pneumonia-618.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-618.png,Viral Pneumonia/images/Viral Pneumonia-618.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-618.png,Viral Pneumonia/masks/Viral Pneumonia-618.png,True,False,299,299,L,91.86,90d1af8f4df0d21daab17226c7ba80b0
Viral Pneumonia-619.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-619.png,Viral Pneumonia/images/Viral Pneumonia-619.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-619.png,Viral Pneumonia/masks/Viral Pneumonia-619.png,True,False,299,299,L,109.32,31d214cabad0dbee907c1d9a13670328
Viral Pneumonia-62.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-62.png,Viral Pneumonia/images/Viral Pneumonia-62.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-62.png,Viral Pneumonia/masks/Viral Pneumonia-62.png,True,False,299,299,L,117.38,58eb05f908914169850509c0e9e098cc
Viral Pneumonia-620.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-620.png,Viral Pneumonia/images/Viral Pneumonia-620.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-620.png,Viral Pneumonia/masks/Viral Pneumonia-620.png,True,False,299,299,L,111.42,df0200634da87966eeb0c540091dc1e5
Viral Pneumonia-621.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-621.png,Viral Pneumonia/images/Viral Pneumonia-621.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-621.png,Viral Pneumonia/masks/Viral Pneumonia-621.png,True,False,299,299,L,87.98,22d9b129dba6684181f844fb8e31d30f
Viral Pneumonia-622.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-622.png,Viral Pneumonia/images/Viral Pneumonia-622.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-622.png,Viral Pneumonia/masks/Viral Pneumonia-622.png,True,False,299,299,L,133.1,19d8eb6c481718d3be95d7b737ecde9f
Viral Pneumonia-623.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-623.png,Viral Pneumonia/images/Viral Pneumonia-623.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-623.png,Viral Pneumonia/masks/Viral Pneumonia-623.png,True,False,299,299,L,114.73,a4767f46d31a23816984e42b7e8564c4
Viral Pneumonia-624.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-624.png,Viral Pneumonia/images/Viral Pneumonia-624.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-624.png,Viral Pneumonia/masks/Viral Pneumonia-624.png,True,False,299,299,RGB,135.53,bfade0c5a162573ac0a059af954859d3
Viral Pneumonia-625.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-625.png,Viral Pneumonia/images/Viral Pneumonia-625.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-625.png,Viral Pneumonia/masks/Viral Pneumonia-625.png,True,False,299,299,L,73.39,fa06328e34a8e7e741c84da6c9efa57f
Viral Pneumonia-626.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-626.png,Viral Pneumonia/images/Viral Pneumonia-626.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-626.png,Viral Pneumonia/masks/Viral Pneumonia-626.png,True,False,299,299,RGB,107.23,05483d82f3517772a4291a560c77d9ef
Viral Pneumonia-627.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-627.png,Viral Pneumonia/images/Viral Pneumonia-627.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-627.png,Viral Pneumonia/masks/Viral Pneumonia-627.png,True,False,299,299,L,127.76,acf9c795e8acaea750cce804708a32ca
Viral Pneumonia-628.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-628.png,Viral Pneumonia/images/Viral Pneumonia-628.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-628.png,Viral Pneumonia/masks/Viral Pneumonia-628.png,True,False,299,299,RGB,128.35,935c69f92a555217741c64cf6e3b46d6
Viral Pneumonia-629.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-629.png,Viral Pneumonia/images/Viral Pneumonia-629.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-629.png,Viral Pneumonia/masks/Viral Pneumonia-629.png,True,False,299,299,L,96.66,e64cf6473bbc9dcc8a8844e710595637
Viral Pneumonia-63.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-63.png,Viral Pneumonia/images/Viral Pneumonia-63.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-63.png,Viral Pneumonia/masks/Viral Pneumonia-63.png,True,False,299,299,L,122.78,fa61359fbbf39d5277b052d53cbd4ed9
Viral Pneumonia-630.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-630.png,Viral Pneumonia/images/Viral Pneumonia-630.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-630.png,Viral Pneumonia/masks/Viral Pneumonia-630.png,True,False,299,299,L,142.7,d27f2f8cf10b2ffeb043d862c74b377e
Viral Pneumonia-631.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-631.png,Viral Pneumonia/images/Viral Pneumonia-631.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-631.png,Viral Pneumonia/masks/Viral Pneumonia-631.png,True,False,299,299,L,113.4,a3e5a58b4c85cfe8c95e6df708b2743f
Viral Pneumonia-632.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-632.png,Viral Pneumonia/images/Viral Pneumonia-632.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-632.png,Viral Pneumonia/masks/Viral Pneumonia-632.png,True,False,299,299,L,116.0,919f116c545ecc9c4434e1bbe6d0d03e
Viral Pneumonia-633.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-633.png,Viral Pneumonia/images/Viral Pneumonia-633.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-633.png,Viral Pneumonia/masks/Viral Pneumonia-633.png,True,False,299,299,L,137.63,b7a4b044d0da5b7d1be1f8686a0a1fd9
Viral Pneumonia-634.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-634.png,Viral Pneumonia/images/Viral Pneumonia-634.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-634.png,Viral Pneumonia/masks/Viral Pneumonia-634.png,True,False,299,299,L,123.41,0d2ab20eefb7755b877b39929fdb2eee
Viral Pneumonia-635.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-635.png,Viral Pneumonia/images/Viral Pneumonia-635.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-635.png,Viral Pneumonia/masks/Viral Pneumonia-635.png,True,False,299,299,RGB,107.93,4d5a993775bd12af7ce514a15e98bfd4
Viral Pneumonia-636.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-636.png,Viral Pneumonia/images/Viral Pneumonia-636.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-636.png,Viral Pneumonia/masks/Viral Pneumonia-636.png,True,False,299,299,RGB,119.37,4353339fdef5de6883be2d3638f36ff9
Viral Pneumonia-637.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-637.png,Viral Pneumonia/images/Viral Pneumonia-637.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-637.png,Viral Pneumonia/masks/Viral Pneumonia-637.png,True,False,299,299,L,116.96,449611b7a25b35285f7bb94c16b2b5d7
Viral Pneumonia-638.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-638.png,Viral Pneumonia/images/Viral Pneumonia-638.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-638.png,Viral Pneumonia/masks/Viral Pneumonia-638.png,True,False,299,299,L,114.11,77bf8aaf5f63b19c22d3f481347c366f
Viral Pneumonia-639.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-639.png,Viral Pneumonia/images/Viral Pneumonia-639.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-639.png,Viral Pneumonia/masks/Viral Pneumonia-639.png,True,False,299,299,L,101.54,f7ca4a4c35d3f4598998d2b7a4ec9f62
Viral Pneumonia-64.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-64.png,Viral Pneumonia/images/Viral Pneumonia-64.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-64.png,Viral Pneumonia/masks/Viral Pneumonia-64.png,True,False,299,299,L,127.32,5186f3d91f2153adf39c1b9a78ae24bd
Viral Pneumonia-640.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-640.png,Viral Pneumonia/images/Viral Pneumonia-640.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-640.png,Viral Pneumonia/masks/Viral Pneumonia-640.png,True,False,299,299,L,95.4,fec50c0fe655c458057eea545bd3bd48
Viral Pneumonia-641.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-641.png,Viral Pneumonia/images/Viral Pneumonia-641.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-641.png,Viral Pneumonia/masks/Viral Pneumonia-641.png,True,False,299,299,L,138.39,0a6e443f4fda0ae3133fc873564d8f33
Viral Pneumonia-642.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-642.png,Viral Pneumonia/images/Viral Pneumonia-642.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-642.png,Viral Pneumonia/masks/Viral Pneumonia-642.png,True,False,299,299,L,99.88,c2f5de109d09a7f682f7de67040f01bd
Viral Pneumonia-643.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-643.png,Viral Pneumonia/images/Viral Pneumonia-643.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-643.png,Viral Pneumonia/masks/Viral Pneumonia-643.png,True,False,299,299,L,165.55,0fc25b44bb1d0d4a3199fdd081ebb6ef
Viral Pneumonia-644.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-644.png,Viral Pneumonia/images/Viral Pneumonia-644.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-644.png,Viral Pneumonia/masks/Viral Pneumonia-644.png,True,False,299,299,L,144.24,086864208a48ab9e377a57ff0793f5a2
Viral Pneumonia-645.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-645.png,Viral Pneumonia/images/Viral Pneumonia-645.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-645.png,Viral Pneumonia/masks/Viral Pneumonia-645.png,True,False,299,299,L,163.1,c436ccdc342bbe01fbe9e2d470ec88ea
Viral Pneumonia-646.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-646.png,Viral Pneumonia/images/Viral Pneumonia-646.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-646.png,Viral Pneumonia/masks/Viral Pneumonia-646.png,True,False,299,299,RGB,132.0,e483fc3db275dfd79390dfd3f5a13a63
Viral Pneumonia-647.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-647.png,Viral Pneumonia/images/Viral Pneumonia-647.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-647.png,Viral Pneumonia/masks/Viral Pneumonia-647.png,True,False,299,299,L,103.26,6fde6df5e9fd560ddeb036ec57edcda7
Viral Pneumonia-648.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-648.png,Viral Pneumonia/images/Viral Pneumonia-648.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-648.png,Viral Pneumonia/masks/Viral Pneumonia-648.png,True,False,299,299,RGB,122.66,31fe27c96d0b201bd09c0bbefb3ba296
Viral Pneumonia-649.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-649.png,Viral Pneumonia/images/Viral Pneumonia-649.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-649.png,Viral Pneumonia/masks/Viral Pneumonia-649.png,True,False,299,299,L,129.47,a299346c764a3af5814f7421d68914f6
Viral Pneumonia-65.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-65.png,Viral Pneumonia/images/Viral Pneumonia-65.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-65.png,Viral Pneumonia/masks/Viral Pneumonia-65.png,True,False,299,299,L,79.63,7e99e68a0f8d8abe15332338279ed9d6
Viral Pneumonia-650.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-650.png,Viral Pneumonia/images/Viral Pneumonia-650.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-650.png,Viral Pneumonia/masks/Viral Pneumonia-650.png,True,False,299,299,L,114.78,958780708464a83f290c2d21bab02931
Viral Pneumonia-651.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-651.png,Viral Pneumonia/images/Viral Pneumonia-651.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-651.png,Viral Pneumonia/masks/Viral Pneumonia-651.png,True,False,299,299,RGB,152.77,3e18677576128a3260be4529709a4536
Viral Pneumonia-652.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-652.png,Viral Pneumonia/images/Viral Pneumonia-652.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-652.png,Viral Pneumonia/masks/Viral Pneumonia-652.png,True,False,299,299,RGB,122.81,631bd1f5fd3bf9c4629754b4644513aa
Viral Pneumonia-653.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-653.png,Viral Pneumonia/images/Viral Pneumonia-653.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-653.png,Viral Pneumonia/masks/Viral Pneumonia-653.png,True,False,299,299,L,66.19,2af72b17319b68ae09f11e63897a9a52
Viral Pneumonia-654.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-654.png,Viral Pneumonia/images/Viral Pneumonia-654.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-654.png,Viral Pneumonia/masks/Viral Pneumonia-654.png,True,False,299,299,RGB,146.03,6b214b743315dad58470f051d92edd25
Viral Pneumonia-655.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-655.png,Viral Pneumonia/images/Viral Pneumonia-655.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-655.png,Viral Pneumonia/masks/Viral Pneumonia-655.png,True,False,299,299,RGB,122.95,6b3a31ff3a367a8bc7438cd955d324c2
Viral Pneumonia-656.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-656.png,Viral Pneumonia/images/Viral Pneumonia-656.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-656.png,Viral Pneumonia/masks/Viral Pneumonia-656.png,True,False,299,299,L,95.74,7f03c45307d1309fb62018a723972692
Viral Pneumonia-657.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-657.png,Viral Pneumonia/images/Viral Pneumonia-657.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-657.png,Viral Pneumonia/masks/Viral Pneumonia-657.png,True,False,299,299,L,80.26,537d197b6d86d5dcd623c1958caa14d9
Viral Pneumonia-658.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-658.png,Viral Pneumonia/images/Viral Pneumonia-658.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-658.png,Viral Pneumonia/masks/Viral Pneumonia-658.png,True,False,299,299,L,108.7,c023c32e72b81eccfdd40596914586f1
Viral Pneumonia-659.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-659.png,Viral Pneumonia/images/Viral Pneumonia-659.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-659.png,Viral Pneumonia/masks/Viral Pneumonia-659.png,True,False,299,299,L,117.71,fce5333fe0ed16f88563ac0b627cfd0c
Viral Pneumonia-66.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-66.png,Viral Pneumonia/images/Viral Pneumonia-66.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-66.png,Viral Pneumonia/masks/Viral Pneumonia-66.png,True,False,299,299,L,79.24,b03139f4a37133cfe0b12c9ca87e61e6
Viral Pneumonia-660.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-660.png,Viral Pneumonia/images/Viral Pneumonia-660.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-660.png,Viral Pneumonia/masks/Viral Pneumonia-660.png,True,False,299,299,L,114.66,849513f81fc482053b55c0ad76ba5d1e
Viral Pneumonia-661.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-661.png,Viral Pneumonia/images/Viral Pneumonia-661.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-661.png,Viral Pneumonia/masks/Viral Pneumonia-661.png,True,False,299,299,L,114.03,66a636692e216bc7dc03aea416d934b0
Viral Pneumonia-662.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-662.png,Viral Pneumonia/images/Viral Pneumonia-662.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-662.png,Viral Pneumonia/masks/Viral Pneumonia-662.png,True,False,299,299,L,87.97,a88159aec4a16b1b620ff4d90eea7477
Viral Pneumonia-663.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-663.png,Viral Pneumonia/images/Viral Pneumonia-663.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-663.png,Viral Pneumonia/masks/Viral Pneumonia-663.png,True,False,299,299,RGB,115.04,21b06e069fb97bb5400b3ab38c17dee6
Viral Pneumonia-664.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-664.png,Viral Pneumonia/images/Viral Pneumonia-664.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-664.png,Viral Pneumonia/masks/Viral Pneumonia-664.png,True,False,299,299,L,99.46,bba2a2c2068556f7eaca590a8b8d116f
Viral Pneumonia-665.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-665.png,Viral Pneumonia/images/Viral Pneumonia-665.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-665.png,Viral Pneumonia/masks/Viral Pneumonia-665.png,True,False,299,299,L,117.37,57461ae76b315b26e82c1a178904eecc
Viral Pneumonia-666.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-666.png,Viral Pneumonia/images/Viral Pneumonia-666.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-666.png,Viral Pneumonia/masks/Viral Pneumonia-666.png,True,False,299,299,L,105.94,7297bad3dc63f594309b4a79ae61f19d
Viral Pneumonia-667.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-667.png,Viral Pneumonia/images/Viral Pneumonia-667.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-667.png,Viral Pneumonia/masks/Viral Pneumonia-667.png,True,False,299,299,L,104.87,e572a591d8c0c952a4a4b5fd62971364
Viral Pneumonia-668.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-668.png,Viral Pneumonia/images/Viral Pneumonia-668.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-668.png,Viral Pneumonia/masks/Viral Pneumonia-668.png,True,False,299,299,L,128.0,e029ec506f085490053a5542061fade0
Viral Pneumonia-669.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-669.png,Viral Pneumonia/images/Viral Pneumonia-669.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-669.png,Viral Pneumonia/masks/Viral Pneumonia-669.png,True,False,299,299,L,159.98,5918b1699641a7fa4490e192d2913298
Viral Pneumonia-67.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-67.png,Viral Pneumonia/images/Viral Pneumonia-67.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-67.png,Viral Pneumonia/masks/Viral Pneumonia-67.png,True,False,299,299,L,111.24,0c6ecad9ec7ffc1524ea964b8dc2bd50
Viral Pneumonia-670.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-670.png,Viral Pneumonia/images/Viral Pneumonia-670.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-670.png,Viral Pneumonia/masks/Viral Pneumonia-670.png,True,False,299,299,L,114.06,6f943b9f133e50b168554f54b4d8a9ce
Viral Pneumonia-671.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-671.png,Viral Pneumonia/images/Viral Pneumonia-671.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-671.png,Viral Pneumonia/masks/Viral Pneumonia-671.png,True,False,299,299,L,120.61,0a7eb660d003e7319fde73346b0bf279
Viral Pneumonia-672.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-672.png,Viral Pneumonia/images/Viral Pneumonia-672.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-672.png,Viral Pneumonia/masks/Viral Pneumonia-672.png,True,False,299,299,L,129.54,c5cb9a93fcae0fc6a6344d4094cca092
Viral Pneumonia-673.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-673.png,Viral Pneumonia/images/Viral Pneumonia-673.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-673.png,Viral Pneumonia/masks/Viral Pneumonia-673.png,True,False,299,299,RGB,120.82,ebe5d22f45005d75a18378bcc40e7944
Viral Pneumonia-674.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-674.png,Viral Pneumonia/images/Viral Pneumonia-674.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-674.png,Viral Pneumonia/masks/Viral Pneumonia-674.png,True,False,299,299,L,142.56,80ae704a85e5f5392cb70f1e3de8bc13
Viral Pneumonia-675.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-675.png,Viral Pneumonia/images/Viral Pneumonia-675.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-675.png,Viral Pneumonia/masks/Viral Pneumonia-675.png,True,False,299,299,L,102.69,05731d6427156aa8a7282f77b9e29945
Viral Pneumonia-676.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-676.png,Viral Pneumonia/images/Viral Pneumonia-676.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-676.png,Viral Pneumonia/masks/Viral Pneumonia-676.png,True,False,299,299,L,108.78,b9457592ea501599cd3d9aba67d0136a
Viral Pneumonia-677.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-677.png,Viral Pneumonia/images/Viral Pneumonia-677.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-677.png,Viral Pneumonia/masks/Viral Pneumonia-677.png,True,False,299,299,L,124.34,3c461c26f1ab441852afeb8454c27abe
Viral Pneumonia-678.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-678.png,Viral Pneumonia/images/Viral Pneumonia-678.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-678.png,Viral Pneumonia/masks/Viral Pneumonia-678.png,True,False,299,299,L,131.34,ef1b470822c8066ec16b9a2fd9e3a461
Viral Pneumonia-679.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-679.png,Viral Pneumonia/images/Viral Pneumonia-679.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-679.png,Viral Pneumonia/masks/Viral Pneumonia-679.png,True,False,299,299,L,109.85,dcc6046616c8504642f7018b7c69f551
Viral Pneumonia-68.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-68.png,Viral Pneumonia/images/Viral Pneumonia-68.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-68.png,Viral Pneumonia/masks/Viral Pneumonia-68.png,True,False,299,299,L,136.24,b2b27c14efd95435f8924b4f444d8e33
Viral Pneumonia-680.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-680.png,Viral Pneumonia/images/Viral Pneumonia-680.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-680.png,Viral Pneumonia/masks/Viral Pneumonia-680.png,True,False,299,299,L,157.07,17cf87907499ffd438330983209c270c
Viral Pneumonia-681.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-681.png,Viral Pneumonia/images/Viral Pneumonia-681.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-681.png,Viral Pneumonia/masks/Viral Pneumonia-681.png,True,False,299,299,L,168.12,3993d2a8b9f088798307113d4ba789d6
Viral Pneumonia-682.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-682.png,Viral Pneumonia/images/Viral Pneumonia-682.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-682.png,Viral Pneumonia/masks/Viral Pneumonia-682.png,True,False,299,299,L,150.96,fd2a02309daaff6f1aa9d6d55e06fa21
Viral Pneumonia-683.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-683.png,Viral Pneumonia/images/Viral Pneumonia-683.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-683.png,Viral Pneumonia/masks/Viral Pneumonia-683.png,True,False,299,299,L,115.82,86417fb3a5c80d4297a07edd50e06956
Viral Pneumonia-684.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-684.png,Viral Pneumonia/images/Viral Pneumonia-684.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-684.png,Viral Pneumonia/masks/Viral Pneumonia-684.png,True,False,299,299,L,105.07,8fc21ce0f8bbc1e1f2ece9a3ef9643b8
Viral Pneumonia-685.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-685.png,Viral Pneumonia/images/Viral Pneumonia-685.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-685.png,Viral Pneumonia/masks/Viral Pneumonia-685.png,True,False,299,299,L,148.76,c0898ebe53d958454cdcf883720f8a80
Viral Pneumonia-686.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-686.png,Viral Pneumonia/images/Viral Pneumonia-686.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-686.png,Viral Pneumonia/masks/Viral Pneumonia-686.png,True,False,299,299,L,131.11,4858f489e69a00854c8e6994e3de5665
Viral Pneumonia-687.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-687.png,Viral Pneumonia/images/Viral Pneumonia-687.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-687.png,Viral Pneumonia/masks/Viral Pneumonia-687.png,True,False,299,299,L,142.22,73f20555ab50abfc02be6050aa8297b5
Viral Pneumonia-688.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-688.png,Viral Pneumonia/images/Viral Pneumonia-688.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-688.png,Viral Pneumonia/masks/Viral Pneumonia-688.png,True,False,299,299,RGB,143.97,3f31b77d859dc4f6f35b5573a86e346a
Viral Pneumonia-689.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-689.png,Viral Pneumonia/images/Viral Pneumonia-689.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-689.png,Viral Pneumonia/masks/Viral Pneumonia-689.png,True,False,299,299,L,114.16,bb4b2160d153aaae5821ec26c550c89a
Viral Pneumonia-69.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-69.png,Viral Pneumonia/images/Viral Pneumonia-69.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-69.png,Viral Pneumonia/masks/Viral Pneumonia-69.png,True,False,299,299,L,121.01,46f95b878595c9aa3a14a43144eedf80
Viral Pneumonia-690.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-690.png,Viral Pneumonia/images/Viral Pneumonia-690.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-690.png,Viral Pneumonia/masks/Viral Pneumonia-690.png,True,False,299,299,L,113.06,90a838c98e5538f7685d36eec14b288d
Viral Pneumonia-691.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-691.png,Viral Pneumonia/images/Viral Pneumonia-691.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-691.png,Viral Pneumonia/masks/Viral Pneumonia-691.png,True,False,299,299,RGB,128.63,c3f697856c8d9eaefa64c46acbfc6179
Viral Pneumonia-692.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-692.png,Viral Pneumonia/images/Viral Pneumonia-692.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-692.png,Viral Pneumonia/masks/Viral Pneumonia-692.png,True,False,299,299,L,100.64,b1247f3cd8f204dbe764920958219fc9
Viral Pneumonia-693.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-693.png,Viral Pneumonia/images/Viral Pneumonia-693.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-693.png,Viral Pneumonia/masks/Viral Pneumonia-693.png,True,False,299,299,RGB,143.6,c6b06fcda2a37d99bd091c2538e4230a
Viral Pneumonia-694.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-694.png,Viral Pneumonia/images/Viral Pneumonia-694.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-694.png,Viral Pneumonia/masks/Viral Pneumonia-694.png,True,False,299,299,L,109.16,7ce837c09882798543f11f13ddb02d1d
Viral Pneumonia-695.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-695.png,Viral Pneumonia/images/Viral Pneumonia-695.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-695.png,Viral Pneumonia/masks/Viral Pneumonia-695.png,True,False,299,299,L,115.14,31150ab93054dbc9764c3adbedb8380b
Viral Pneumonia-696.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-696.png,Viral Pneumonia/images/Viral Pneumonia-696.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-696.png,Viral Pneumonia/masks/Viral Pneumonia-696.png,True,False,299,299,L,157.2,80ef8d3feb775e4e3caa0c9b068018c3
Viral Pneumonia-697.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-697.png,Viral Pneumonia/images/Viral Pneumonia-697.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-697.png,Viral Pneumonia/masks/Viral Pneumonia-697.png,True,False,299,299,L,153.64,19b258a7f5f53432bbdc88553b77d38c
Viral Pneumonia-698.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-698.png,Viral Pneumonia/images/Viral Pneumonia-698.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-698.png,Viral Pneumonia/masks/Viral Pneumonia-698.png,True,False,299,299,L,135.43,d7827433e03fce0835e5f20cda77bcec
Viral Pneumonia-699.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-699.png,Viral Pneumonia/images/Viral Pneumonia-699.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-699.png,Viral Pneumonia/masks/Viral Pneumonia-699.png,True,False,299,299,L,97.9,0ed4fd50bfd79090daa40b04665ddb50
Viral Pneumonia-7.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-7.png,Viral Pneumonia/images/Viral Pneumonia-7.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-7.png,Viral Pneumonia/masks/Viral Pneumonia-7.png,True,False,299,299,L,115.84,c9fb59bf00fd9d3070ed616ce2a5b029
Viral Pneumonia-70.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-70.png,Viral Pneumonia/images/Viral Pneumonia-70.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-70.png,Viral Pneumonia/masks/Viral Pneumonia-70.png,True,False,299,299,L,101.48,65030adc92d0b634921782647adda586
Viral Pneumonia-700.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-700.png,Viral Pneumonia/images/Viral Pneumonia-700.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-700.png,Viral Pneumonia/masks/Viral Pneumonia-700.png,True,False,299,299,L,137.79,d0e7128f02ed8a2e99a6c65ee01b2bda
Viral Pneumonia-701.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-701.png,Viral Pneumonia/images/Viral Pneumonia-701.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-701.png,Viral Pneumonia/masks/Viral Pneumonia-701.png,True,False,299,299,L,128.3,f37112f0bbc02a759c4d1e1157ddc4d8
Viral Pneumonia-702.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-702.png,Viral Pneumonia/images/Viral Pneumonia-702.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-702.png,Viral Pneumonia/masks/Viral Pneumonia-702.png,True,False,299,299,L,111.21,0b50372f870c7fd3bea520e85186b015
Viral Pneumonia-703.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-703.png,Viral Pneumonia/images/Viral Pneumonia-703.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-703.png,Viral Pneumonia/masks/Viral Pneumonia-703.png,True,False,299,299,L,124.52,577ad585f91753bcdcc1a6f1116850d2
Viral Pneumonia-704.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-704.png,Viral Pneumonia/images/Viral Pneumonia-704.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-704.png,Viral Pneumonia/masks/Viral Pneumonia-704.png,True,False,299,299,L,128.38,57e49aeec1d571c524ccaa9631c4d74d
Viral Pneumonia-705.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-705.png,Viral Pneumonia/images/Viral Pneumonia-705.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-705.png,Viral Pneumonia/masks/Viral Pneumonia-705.png,True,False,299,299,L,111.52,33c58fbcce257bcb9da2db9ffb527d59
Viral Pneumonia-706.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-706.png,Viral Pneumonia/images/Viral Pneumonia-706.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-706.png,Viral Pneumonia/masks/Viral Pneumonia-706.png,True,False,299,299,L,143.88,930a41cc3d8966106f0b9b92d6673ae0
Viral Pneumonia-707.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-707.png,Viral Pneumonia/images/Viral Pneumonia-707.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-707.png,Viral Pneumonia/masks/Viral Pneumonia-707.png,True,False,299,299,L,128.17,ed9062d313cd0a812b1c963d250f7e3e
Viral Pneumonia-708.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-708.png,Viral Pneumonia/images/Viral Pneumonia-708.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-708.png,Viral Pneumonia/masks/Viral Pneumonia-708.png,True,False,299,299,L,131.57,3cd91196e57375e96835373a5ce63b39
Viral Pneumonia-709.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-709.png,Viral Pneumonia/images/Viral Pneumonia-709.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-709.png,Viral Pneumonia/masks/Viral Pneumonia-709.png,True,False,299,299,L,146.76,d5b6c1799ede8fb4c382f970be24ae7e
Viral Pneumonia-71.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-71.png,Viral Pneumonia/images/Viral Pneumonia-71.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-71.png,Viral Pneumonia/masks/Viral Pneumonia-71.png,True,False,299,299,L,118.01,2fc8823e8d6ecd73e714a50991243f89
Viral Pneumonia-710.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-710.png,Viral Pneumonia/images/Viral Pneumonia-710.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-710.png,Viral Pneumonia/masks/Viral Pneumonia-710.png,True,False,299,299,L,125.57,fa0caa0d2fb208cf083875462ea27603
Viral Pneumonia-711.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-711.png,Viral Pneumonia/images/Viral Pneumonia-711.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-711.png,Viral Pneumonia/masks/Viral Pneumonia-711.png,True,False,299,299,L,141.26,87d290c073f433b67ee05e03422f382b
Viral Pneumonia-712.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-712.png,Viral Pneumonia/images/Viral Pneumonia-712.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-712.png,Viral Pneumonia/masks/Viral Pneumonia-712.png,True,False,299,299,L,133.64,7aae9491342337f24887b36819f1dd6e
Viral Pneumonia-713.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-713.png,Viral Pneumonia/images/Viral Pneumonia-713.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-713.png,Viral Pneumonia/masks/Viral Pneumonia-713.png,True,False,299,299,L,113.66,fedbf1e01ccfae1a9c3ce10f3d1eef75
Viral Pneumonia-714.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-714.png,Viral Pneumonia/images/Viral Pneumonia-714.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-714.png,Viral Pneumonia/masks/Viral Pneumonia-714.png,True,False,299,299,L,104.43,5b47589ff8d9be59e0ededd525c821b5
Viral Pneumonia-715.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-715.png,Viral Pneumonia/images/Viral Pneumonia-715.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-715.png,Viral Pneumonia/masks/Viral Pneumonia-715.png,True,False,299,299,L,146.91,da3116be3b14364563c04232fadcad36
Viral Pneumonia-716.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-716.png,Viral Pneumonia/images/Viral Pneumonia-716.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-716.png,Viral Pneumonia/masks/Viral Pneumonia-716.png,True,False,299,299,L,144.02,dd7aa310e228b6abf86d8cd823094b8c
Viral Pneumonia-717.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-717.png,Viral Pneumonia/images/Viral Pneumonia-717.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-717.png,Viral Pneumonia/masks/Viral Pneumonia-717.png,True,False,299,299,L,138.02,68d8c1e5d36fa2a231254b2cc4672a09
Viral Pneumonia-718.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-718.png,Viral Pneumonia/images/Viral Pneumonia-718.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-718.png,Viral Pneumonia/masks/Viral Pneumonia-718.png,True,False,299,299,L,144.07,95a12a5b47b733ab5582e74b5ad83f7a
Viral Pneumonia-719.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-719.png,Viral Pneumonia/images/Viral Pneumonia-719.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-719.png,Viral Pneumonia/masks/Viral Pneumonia-719.png,True,False,299,299,L,148.08,27a231b1b943bded37513c2fb3eac837
Viral Pneumonia-72.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-72.png,Viral Pneumonia/images/Viral Pneumonia-72.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-72.png,Viral Pneumonia/masks/Viral Pneumonia-72.png,True,False,299,299,L,115.21,1dd24be7f512fa767127a21904d2090b
Viral Pneumonia-720.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-720.png,Viral Pneumonia/images/Viral Pneumonia-720.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-720.png,Viral Pneumonia/masks/Viral Pneumonia-720.png,True,False,299,299,L,111.2,3cecc31afc812b949739f85a83f2cd3e
Viral Pneumonia-721.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-721.png,Viral Pneumonia/images/Viral Pneumonia-721.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-721.png,Viral Pneumonia/masks/Viral Pneumonia-721.png,True,False,299,299,L,107.41,88af3c70649af84d7a32faa49fbe0698
Viral Pneumonia-722.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-722.png,Viral Pneumonia/images/Viral Pneumonia-722.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-722.png,Viral Pneumonia/masks/Viral Pneumonia-722.png,True,False,299,299,L,121.47,c8280784b61211e67a7f777059477355
Viral Pneumonia-723.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-723.png,Viral Pneumonia/images/Viral Pneumonia-723.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-723.png,Viral Pneumonia/masks/Viral Pneumonia-723.png,True,False,299,299,L,122.26,26efe14fd3c38bbb737fad1f9ee711f8
Viral Pneumonia-724.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-724.png,Viral Pneumonia/images/Viral Pneumonia-724.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-724.png,Viral Pneumonia/masks/Viral Pneumonia-724.png,True,False,299,299,L,111.02,3db81dc16e4ceac7ae8a39eb3c7df06f
Viral Pneumonia-725.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-725.png,Viral Pneumonia/images/Viral Pneumonia-725.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-725.png,Viral Pneumonia/masks/Viral Pneumonia-725.png,True,False,299,299,L,97.76,c16ef59c2d9494f8a39078e23768eb19
Viral Pneumonia-726.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-726.png,Viral Pneumonia/images/Viral Pneumonia-726.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-726.png,Viral Pneumonia/masks/Viral Pneumonia-726.png,True,False,299,299,L,108.19,c589241cc9aa52ec1c9b4f0c82210549
Viral Pneumonia-727.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-727.png,Viral Pneumonia/images/Viral Pneumonia-727.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-727.png,Viral Pneumonia/masks/Viral Pneumonia-727.png,True,False,299,299,L,116.67,c67b12ded169868cf965ac9eb2e39e13
Viral Pneumonia-728.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-728.png,Viral Pneumonia/images/Viral Pneumonia-728.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-728.png,Viral Pneumonia/masks/Viral Pneumonia-728.png,True,False,299,299,L,67.79,992f4338a6531639092644d96b2c5f66
Viral Pneumonia-729.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-729.png,Viral Pneumonia/images/Viral Pneumonia-729.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-729.png,Viral Pneumonia/masks/Viral Pneumonia-729.png,True,False,299,299,L,124.77,b77c09f2ac141e35bb11698d1de3ab59
Viral Pneumonia-73.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-73.png,Viral Pneumonia/images/Viral Pneumonia-73.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-73.png,Viral Pneumonia/masks/Viral Pneumonia-73.png,True,False,299,299,L,133.58,9e470ef79e7c99e822f31216ba1d7ea1
Viral Pneumonia-730.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-730.png,Viral Pneumonia/images/Viral Pneumonia-730.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-730.png,Viral Pneumonia/masks/Viral Pneumonia-730.png,True,False,299,299,L,155.13,b0f58b75b8729e60555828607a9a908e
Viral Pneumonia-731.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-731.png,Viral Pneumonia/images/Viral Pneumonia-731.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-731.png,Viral Pneumonia/masks/Viral Pneumonia-731.png,True,False,299,299,L,114.99,5a46188071547a14fbdbbaa22e844aea
Viral Pneumonia-732.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-732.png,Viral Pneumonia/images/Viral Pneumonia-732.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-732.png,Viral Pneumonia/masks/Viral Pneumonia-732.png,True,False,299,299,L,123.6,af48c228101b2843ea2a1faa2ae9e7fb
Viral Pneumonia-733.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-733.png,Viral Pneumonia/images/Viral Pneumonia-733.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-733.png,Viral Pneumonia/masks/Viral Pneumonia-733.png,True,False,299,299,L,99.5,5fb8b2cd28e23f6f5701efc448d9fbf9
Viral Pneumonia-734.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-734.png,Viral Pneumonia/images/Viral Pneumonia-734.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-734.png,Viral Pneumonia/masks/Viral Pneumonia-734.png,True,False,299,299,L,135.65,5bd5ac2f198c0688e6c9cea8c6952ffd
Viral Pneumonia-735.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-735.png,Viral Pneumonia/images/Viral Pneumonia-735.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-735.png,Viral Pneumonia/masks/Viral Pneumonia-735.png,True,False,299,299,L,135.58,645e50c845ffa8c2bcadcfdb5bc4b5e3
Viral Pneumonia-736.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-736.png,Viral Pneumonia/images/Viral Pneumonia-736.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-736.png,Viral Pneumonia/masks/Viral Pneumonia-736.png,True,False,299,299,L,106.89,3010152069e13223377c025bda47230b
Viral Pneumonia-737.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-737.png,Viral Pneumonia/images/Viral Pneumonia-737.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-737.png,Viral Pneumonia/masks/Viral Pneumonia-737.png,True,False,299,299,L,111.68,afd343767e20181bdf27cd4aacf91b6b
Viral Pneumonia-738.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-738.png,Viral Pneumonia/images/Viral Pneumonia-738.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-738.png,Viral Pneumonia/masks/Viral Pneumonia-738.png,True,False,299,299,L,100.82,9596b2d52ed6a108a2d3c0eae0adac2f
Viral Pneumonia-739.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-739.png,Viral Pneumonia/images/Viral Pneumonia-739.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-739.png,Viral Pneumonia/masks/Viral Pneumonia-739.png,True,False,299,299,L,105.19,8093a2a71ddd77a1c4a1e9e4e62f2bfa
Viral Pneumonia-74.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-74.png,Viral Pneumonia/images/Viral Pneumonia-74.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-74.png,Viral Pneumonia/masks/Viral Pneumonia-74.png,True,False,299,299,L,122.85,1c28d2b6859a90eb21544f885109150d
Viral Pneumonia-740.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-740.png,Viral Pneumonia/images/Viral Pneumonia-740.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-740.png,Viral Pneumonia/masks/Viral Pneumonia-740.png,True,False,299,299,L,125.91,c6b02f768d1d38fedb7091f0b1c6038e
Viral Pneumonia-741.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-741.png,Viral Pneumonia/images/Viral Pneumonia-741.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-741.png,Viral Pneumonia/masks/Viral Pneumonia-741.png,True,False,299,299,L,111.04,dd05421b151ae9b9103863e3f8393ff3
Viral Pneumonia-742.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-742.png,Viral Pneumonia/images/Viral Pneumonia-742.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-742.png,Viral Pneumonia/masks/Viral Pneumonia-742.png,True,False,299,299,RGB,155.09,fa498054e01f5fac9ff4183df70b45a3
Viral Pneumonia-743.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-743.png,Viral Pneumonia/images/Viral Pneumonia-743.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-743.png,Viral Pneumonia/masks/Viral Pneumonia-743.png,True,False,299,299,L,123.71,6ce6d21b347e7063ac620ee2089eb157
Viral Pneumonia-744.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-744.png,Viral Pneumonia/images/Viral Pneumonia-744.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-744.png,Viral Pneumonia/masks/Viral Pneumonia-744.png,True,False,299,299,L,141.19,cf20204c2ac6300f37d35d22c247c1d5
Viral Pneumonia-745.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-745.png,Viral Pneumonia/images/Viral Pneumonia-745.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-745.png,Viral Pneumonia/masks/Viral Pneumonia-745.png,True,False,299,299,RGB,141.01,f64f3a847068ca5c19b427d6227e487a
Viral Pneumonia-746.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-746.png,Viral Pneumonia/images/Viral Pneumonia-746.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-746.png,Viral Pneumonia/masks/Viral Pneumonia-746.png,True,False,299,299,L,123.78,359e658ecbcf6dcb2353996634d08b74
Viral Pneumonia-747.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-747.png,Viral Pneumonia/images/Viral Pneumonia-747.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-747.png,Viral Pneumonia/masks/Viral Pneumonia-747.png,True,False,299,299,L,132.52,da706fba03664c1b1086a522e6e305e5
Viral Pneumonia-748.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-748.png,Viral Pneumonia/images/Viral Pneumonia-748.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-748.png,Viral Pneumonia/masks/Viral Pneumonia-748.png,True,False,299,299,L,106.44,3bcb21eb616cc84b23dfdae3d19d6cf1
Viral Pneumonia-749.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-749.png,Viral Pneumonia/images/Viral Pneumonia-749.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-749.png,Viral Pneumonia/masks/Viral Pneumonia-749.png,True,False,299,299,L,136.82,2f791eb2232171b3567288f7c785063e
Viral Pneumonia-75.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-75.png,Viral Pneumonia/images/Viral Pneumonia-75.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-75.png,Viral Pneumonia/masks/Viral Pneumonia-75.png,True,False,299,299,L,122.85,1c28d2b6859a90eb21544f885109150d
Viral Pneumonia-750.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-750.png,Viral Pneumonia/images/Viral Pneumonia-750.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-750.png,Viral Pneumonia/masks/Viral Pneumonia-750.png,True,False,299,299,RGB,159.17,b52b5b160f0ddcf98b31941328f18dd5
Viral Pneumonia-751.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-751.png,Viral Pneumonia/images/Viral Pneumonia-751.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-751.png,Viral Pneumonia/masks/Viral Pneumonia-751.png,True,False,299,299,L,122.01,7d8f73366a83b7b543cc9957b50a2b20
Viral Pneumonia-752.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-752.png,Viral Pneumonia/images/Viral Pneumonia-752.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-752.png,Viral Pneumonia/masks/Viral Pneumonia-752.png,True,False,299,299,RGB,127.55,18cd0f2d44e6875468c8c601d175ca8a
Viral Pneumonia-753.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-753.png,Viral Pneumonia/images/Viral Pneumonia-753.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-753.png,Viral Pneumonia/masks/Viral Pneumonia-753.png,True,False,299,299,RGB,153.13,5c1260b6900a07d304e760dae2ade3b8
Viral Pneumonia-754.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-754.png,Viral Pneumonia/images/Viral Pneumonia-754.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-754.png,Viral Pneumonia/masks/Viral Pneumonia-754.png,True,False,299,299,L,130.33,91121656c3c383fd56ee7ede50b33cf8
Viral Pneumonia-755.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-755.png,Viral Pneumonia/images/Viral Pneumonia-755.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-755.png,Viral Pneumonia/masks/Viral Pneumonia-755.png,True,False,299,299,L,95.82,7da778d6f8ed3e9b70437f1a2af28f5f
Viral Pneumonia-756.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-756.png,Viral Pneumonia/images/Viral Pneumonia-756.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-756.png,Viral Pneumonia/masks/Viral Pneumonia-756.png,True,False,299,299,L,147.16,e302135f44a3bad2815616f7c60acd49
Viral Pneumonia-757.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-757.png,Viral Pneumonia/images/Viral Pneumonia-757.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-757.png,Viral Pneumonia/masks/Viral Pneumonia-757.png,True,False,299,299,L,138.52,5215c74f09f5b8fb72c819331b42a2a6
Viral Pneumonia-758.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-758.png,Viral Pneumonia/images/Viral Pneumonia-758.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-758.png,Viral Pneumonia/masks/Viral Pneumonia-758.png,True,False,299,299,L,134.2,111f64fee076d023626d3042347e1c91
Viral Pneumonia-759.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-759.png,Viral Pneumonia/images/Viral Pneumonia-759.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-759.png,Viral Pneumonia/masks/Viral Pneumonia-759.png,True,False,299,299,L,145.51,49def0b3c6ae7fb9ae7263cef9b1035d
Viral Pneumonia-76.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-76.png,Viral Pneumonia/images/Viral Pneumonia-76.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-76.png,Viral Pneumonia/masks/Viral Pneumonia-76.png,True,False,299,299,L,113.27,44ee314f77b312f701fb3381bbbb17d4
Viral Pneumonia-760.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-760.png,Viral Pneumonia/images/Viral Pneumonia-760.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-760.png,Viral Pneumonia/masks/Viral Pneumonia-760.png,True,False,299,299,L,167.07,c4c7f1fc7239b71337bed4af77774850
Viral Pneumonia-761.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-761.png,Viral Pneumonia/images/Viral Pneumonia-761.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-761.png,Viral Pneumonia/masks/Viral Pneumonia-761.png,True,False,299,299,L,125.92,97c57bd56bebaec75066342793c88287
Viral Pneumonia-762.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-762.png,Viral Pneumonia/images/Viral Pneumonia-762.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-762.png,Viral Pneumonia/masks/Viral Pneumonia-762.png,True,False,299,299,L,103.86,9ab6673126fad6c86893ebcc7d64fb30
Viral Pneumonia-763.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-763.png,Viral Pneumonia/images/Viral Pneumonia-763.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-763.png,Viral Pneumonia/masks/Viral Pneumonia-763.png,True,False,299,299,L,149.13,67e5ac33be6b68eb44fd74397fa9d2d5
Viral Pneumonia-764.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-764.png,Viral Pneumonia/images/Viral Pneumonia-764.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-764.png,Viral Pneumonia/masks/Viral Pneumonia-764.png,True,False,299,299,RGB,130.47,ca81d5a40d52b29a417dff3bccbc4d67
Viral Pneumonia-765.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-765.png,Viral Pneumonia/images/Viral Pneumonia-765.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-765.png,Viral Pneumonia/masks/Viral Pneumonia-765.png,True,False,299,299,L,120.16,a865a4567c96fb7842529341daf4ad52
Viral Pneumonia-766.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-766.png,Viral Pneumonia/images/Viral Pneumonia-766.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-766.png,Viral Pneumonia/masks/Viral Pneumonia-766.png,True,False,299,299,L,145.73,bc7f49adbaa03cba0f3645b88f047639
Viral Pneumonia-767.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-767.png,Viral Pneumonia/images/Viral Pneumonia-767.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-767.png,Viral Pneumonia/masks/Viral Pneumonia-767.png,True,False,299,299,L,92.45,600fe503e63275240c1fa95b0c0f6126
Viral Pneumonia-768.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-768.png,Viral Pneumonia/images/Viral Pneumonia-768.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-768.png,Viral Pneumonia/masks/Viral Pneumonia-768.png,True,False,299,299,RGB,70.55,17bde0799379ffbcede6a64b73abc990
Viral Pneumonia-769.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-769.png,Viral Pneumonia/images/Viral Pneumonia-769.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-769.png,Viral Pneumonia/masks/Viral Pneumonia-769.png,True,False,299,299,L,105.37,ee8cd0c2aaffe49b6697de41ab8ce577
Viral Pneumonia-77.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-77.png,Viral Pneumonia/images/Viral Pneumonia-77.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-77.png,Viral Pneumonia/masks/Viral Pneumonia-77.png,True,False,299,299,L,134.3,362fd131c97d46a00a059668bd34bdc8
Viral Pneumonia-770.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-770.png,Viral Pneumonia/images/Viral Pneumonia-770.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-770.png,Viral Pneumonia/masks/Viral Pneumonia-770.png,True,False,299,299,RGB,139.36,3478df5d05ba9dd2ecb722fea5526fb8
Viral Pneumonia-771.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-771.png,Viral Pneumonia/images/Viral Pneumonia-771.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-771.png,Viral Pneumonia/masks/Viral Pneumonia-771.png,True,False,299,299,L,129.65,22311dca9f9458b325158530548dcfdc
Viral Pneumonia-772.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-772.png,Viral Pneumonia/images/Viral Pneumonia-772.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-772.png,Viral Pneumonia/masks/Viral Pneumonia-772.png,True,False,299,299,L,169.54,8aea6c9c25cef9d93d1440acb077bef9
Viral Pneumonia-773.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-773.png,Viral Pneumonia/images/Viral Pneumonia-773.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-773.png,Viral Pneumonia/masks/Viral Pneumonia-773.png,True,False,299,299,RGB,113.74,84ae02d329bbd11711f1af4cedaa9045
Viral Pneumonia-774.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-774.png,Viral Pneumonia/images/Viral Pneumonia-774.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-774.png,Viral Pneumonia/masks/Viral Pneumonia-774.png,True,False,299,299,L,129.03,3252495edf5b4587e2b46969c995531e
Viral Pneumonia-775.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-775.png,Viral Pneumonia/images/Viral Pneumonia-775.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-775.png,Viral Pneumonia/masks/Viral Pneumonia-775.png,True,False,299,299,L,162.34,aca9cdceaa10ee28584b0b5b7711b2b9
Viral Pneumonia-776.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-776.png,Viral Pneumonia/images/Viral Pneumonia-776.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-776.png,Viral Pneumonia/masks/Viral Pneumonia-776.png,True,False,299,299,L,121.35,a4a247bc1a1ad8dec437f5552e9b0f95
Viral Pneumonia-777.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-777.png,Viral Pneumonia/images/Viral Pneumonia-777.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-777.png,Viral Pneumonia/masks/Viral Pneumonia-777.png,True,False,299,299,L,137.4,37c6d17d839dddf4ae81fde782098ebc
Viral Pneumonia-778.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-778.png,Viral Pneumonia/images/Viral Pneumonia-778.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-778.png,Viral Pneumonia/masks/Viral Pneumonia-778.png,True,False,299,299,L,132.29,7faf299d7694275f19a7bb09e13e91c6
Viral Pneumonia-779.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-779.png,Viral Pneumonia/images/Viral Pneumonia-779.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-779.png,Viral Pneumonia/masks/Viral Pneumonia-779.png,True,False,299,299,L,118.91,eeb0e231e65ed2f2a0ec1d40cbc3468d
Viral Pneumonia-78.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-78.png,Viral Pneumonia/images/Viral Pneumonia-78.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-78.png,Viral Pneumonia/masks/Viral Pneumonia-78.png,True,False,299,299,L,129.15,694cb99a9c6cfc4630e0677a1ee4540a
Viral Pneumonia-780.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-780.png,Viral Pneumonia/images/Viral Pneumonia-780.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-780.png,Viral Pneumonia/masks/Viral Pneumonia-780.png,True,False,299,299,L,126.48,fb5d18077a6b4df875fa682f6e3f2ba1
Viral Pneumonia-781.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-781.png,Viral Pneumonia/images/Viral Pneumonia-781.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-781.png,Viral Pneumonia/masks/Viral Pneumonia-781.png,True,False,299,299,L,98.97,83170fdaa7d58f598e50e8b9c71fe7c1
Viral Pneumonia-782.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-782.png,Viral Pneumonia/images/Viral Pneumonia-782.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-782.png,Viral Pneumonia/masks/Viral Pneumonia-782.png,True,False,299,299,RGB,117.39,87146cba4f707d35fc82d6de3db9537d
Viral Pneumonia-783.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-783.png,Viral Pneumonia/images/Viral Pneumonia-783.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-783.png,Viral Pneumonia/masks/Viral Pneumonia-783.png,True,False,299,299,L,93.17,e75dcd17c2c610a50076ae105468ce9a
Viral Pneumonia-784.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-784.png,Viral Pneumonia/images/Viral Pneumonia-784.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-784.png,Viral Pneumonia/masks/Viral Pneumonia-784.png,True,False,299,299,L,135.48,9b21f06e4c50efd411d090d37f257512
Viral Pneumonia-785.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-785.png,Viral Pneumonia/images/Viral Pneumonia-785.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-785.png,Viral Pneumonia/masks/Viral Pneumonia-785.png,True,False,299,299,L,117.72,19c7250a815a2b89e6f0e8edea807ec0
Viral Pneumonia-786.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-786.png,Viral Pneumonia/images/Viral Pneumonia-786.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-786.png,Viral Pneumonia/masks/Viral Pneumonia-786.png,True,False,299,299,L,133.64,4e46ccfa2950c9cae48a8b9e37d8954c
Viral Pneumonia-787.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-787.png,Viral Pneumonia/images/Viral Pneumonia-787.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-787.png,Viral Pneumonia/masks/Viral Pneumonia-787.png,True,False,299,299,L,133.9,6b7281b722c1849fb29a45fd4af95396
Viral Pneumonia-788.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-788.png,Viral Pneumonia/images/Viral Pneumonia-788.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-788.png,Viral Pneumonia/masks/Viral Pneumonia-788.png,True,False,299,299,L,97.85,b7c791a0c39f2bbadcc489ac04e8ad7b
Viral Pneumonia-789.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-789.png,Viral Pneumonia/images/Viral Pneumonia-789.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-789.png,Viral Pneumonia/masks/Viral Pneumonia-789.png,True,False,299,299,L,111.7,382907e3cd3f825979ec608858967bb5
Viral Pneumonia-79.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-79.png,Viral Pneumonia/images/Viral Pneumonia-79.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-79.png,Viral Pneumonia/masks/Viral Pneumonia-79.png,True,False,299,299,L,143.51,2a3a1837e4feaf8a25eedc33e0826e6a
Viral Pneumonia-790.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-790.png,Viral Pneumonia/images/Viral Pneumonia-790.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-790.png,Viral Pneumonia/masks/Viral Pneumonia-790.png,True,False,299,299,RGB,157.54,638f40d0cb7de7f96e2e71e69fea755b
Viral Pneumonia-791.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-791.png,Viral Pneumonia/images/Viral Pneumonia-791.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-791.png,Viral Pneumonia/masks/Viral Pneumonia-791.png,True,False,299,299,L,94.75,05c9eae62c2a5751186426fa50a4140b
Viral Pneumonia-792.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-792.png,Viral Pneumonia/images/Viral Pneumonia-792.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-792.png,Viral Pneumonia/masks/Viral Pneumonia-792.png,True,False,299,299,RGB,165.18,5c073c67398c99dfb7fe371a14549916
Viral Pneumonia-793.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-793.png,Viral Pneumonia/images/Viral Pneumonia-793.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-793.png,Viral Pneumonia/masks/Viral Pneumonia-793.png,True,False,299,299,L,118.7,97b6c3d64e5030467753d9bf26050d3e
Viral Pneumonia-794.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-794.png,Viral Pneumonia/images/Viral Pneumonia-794.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-794.png,Viral Pneumonia/masks/Viral Pneumonia-794.png,True,False,299,299,L,110.07,bc7a330788c8270781eb033f7ce7777e
Viral Pneumonia-795.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-795.png,Viral Pneumonia/images/Viral Pneumonia-795.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-795.png,Viral Pneumonia/masks/Viral Pneumonia-795.png,True,False,299,299,L,150.92,142ad54b8eedb2a44599bd5424f79b39
Viral Pneumonia-796.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-796.png,Viral Pneumonia/images/Viral Pneumonia-796.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-796.png,Viral Pneumonia/masks/Viral Pneumonia-796.png,True,False,299,299,L,179.01,0dcff59f11606a24ee1d7264640bfd1a
Viral Pneumonia-797.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-797.png,Viral Pneumonia/images/Viral Pneumonia-797.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-797.png,Viral Pneumonia/masks/Viral Pneumonia-797.png,True,False,299,299,L,113.42,8d26815be4432e59f7024fab62b2cb21
Viral Pneumonia-798.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-798.png,Viral Pneumonia/images/Viral Pneumonia-798.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-798.png,Viral Pneumonia/masks/Viral Pneumonia-798.png,True,False,299,299,L,137.97,a52b9998f342226b5e723b7fb0a58682
Viral Pneumonia-799.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-799.png,Viral Pneumonia/images/Viral Pneumonia-799.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-799.png,Viral Pneumonia/masks/Viral Pneumonia-799.png,True,False,299,299,L,146.94,00effedcf1b4a01b5228660a6dac0e3c
Viral Pneumonia-8.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-8.png,Viral Pneumonia/images/Viral Pneumonia-8.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-8.png,Viral Pneumonia/masks/Viral Pneumonia-8.png,True,False,299,299,L,109.84,31e8a3a80b980ce7c34980975683c9b3
Viral Pneumonia-80.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-80.png,Viral Pneumonia/images/Viral Pneumonia-80.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-80.png,Viral Pneumonia/masks/Viral Pneumonia-80.png,True,False,299,299,L,150.26,d32d83df89145089af6e17c2aad135c3
Viral Pneumonia-800.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-800.png,Viral Pneumonia/images/Viral Pneumonia-800.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-800.png,Viral Pneumonia/masks/Viral Pneumonia-800.png,True,False,299,299,L,84.02,6470f5ab595898d9a69615fd26c4a86c
Viral Pneumonia-801.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-801.png,Viral Pneumonia/images/Viral Pneumonia-801.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-801.png,Viral Pneumonia/masks/Viral Pneumonia-801.png,True,False,299,299,L,84.54,e2fa34e38a4e7813f637482520175e85
Viral Pneumonia-802.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-802.png,Viral Pneumonia/images/Viral Pneumonia-802.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-802.png,Viral Pneumonia/masks/Viral Pneumonia-802.png,True,False,299,299,L,124.88,5206028da8034b44eb84d34a195c98f8
Viral Pneumonia-803.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-803.png,Viral Pneumonia/images/Viral Pneumonia-803.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-803.png,Viral Pneumonia/masks/Viral Pneumonia-803.png,True,False,299,299,RGB,134.41,21627ba7f467c9157b2f56a89b5885be
Viral Pneumonia-804.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-804.png,Viral Pneumonia/images/Viral Pneumonia-804.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-804.png,Viral Pneumonia/masks/Viral Pneumonia-804.png,True,False,299,299,L,96.31,808f41777aa1e4e8a7c0ae7b45728b41
Viral Pneumonia-805.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-805.png,Viral Pneumonia/images/Viral Pneumonia-805.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-805.png,Viral Pneumonia/masks/Viral Pneumonia-805.png,True,False,299,299,L,146.41,6110e4c1204ed22f73dafa8c7f139794
Viral Pneumonia-806.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-806.png,Viral Pneumonia/images/Viral Pneumonia-806.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-806.png,Viral Pneumonia/masks/Viral Pneumonia-806.png,True,False,299,299,RGB,114.52,93fbf617ccfdfdf4034e535e03df1c13
Viral Pneumonia-807.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-807.png,Viral Pneumonia/images/Viral Pneumonia-807.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-807.png,Viral Pneumonia/masks/Viral Pneumonia-807.png,True,False,299,299,L,99.73,d49150421b6e4e990f02e05c4945c27b
Viral Pneumonia-808.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-808.png,Viral Pneumonia/images/Viral Pneumonia-808.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-808.png,Viral Pneumonia/masks/Viral Pneumonia-808.png,True,False,299,299,L,143.77,1b2bbdbbe7514b430436632076315f2e
Viral Pneumonia-809.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-809.png,Viral Pneumonia/images/Viral Pneumonia-809.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-809.png,Viral Pneumonia/masks/Viral Pneumonia-809.png,True,False,299,299,L,118.42,53c823620a1b2ec48427bbfe3d797e67
Viral Pneumonia-81.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-81.png,Viral Pneumonia/images/Viral Pneumonia-81.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-81.png,Viral Pneumonia/masks/Viral Pneumonia-81.png,True,False,299,299,L,137.74,72d4a0d60fd3d3ce1fd4b5fdef2da4c2
Viral Pneumonia-810.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-810.png,Viral Pneumonia/images/Viral Pneumonia-810.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-810.png,Viral Pneumonia/masks/Viral Pneumonia-810.png,True,False,299,299,L,136.74,2dbc9c73e2e3723a398bc5b98275cb4d
Viral Pneumonia-811.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-811.png,Viral Pneumonia/images/Viral Pneumonia-811.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-811.png,Viral Pneumonia/masks/Viral Pneumonia-811.png,True,False,299,299,L,114.87,b1a5e8da6906646a3fc2e73a9cdacb33
Viral Pneumonia-812.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-812.png,Viral Pneumonia/images/Viral Pneumonia-812.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-812.png,Viral Pneumonia/masks/Viral Pneumonia-812.png,True,False,299,299,L,125.5,33d0df08a605b11774d1c4c3105307e6
Viral Pneumonia-813.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-813.png,Viral Pneumonia/images/Viral Pneumonia-813.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-813.png,Viral Pneumonia/masks/Viral Pneumonia-813.png,True,False,299,299,L,78.54,587e9d1fbbd735db8fb34ccc3cc04574
Viral Pneumonia-814.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-814.png,Viral Pneumonia/images/Viral Pneumonia-814.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-814.png,Viral Pneumonia/masks/Viral Pneumonia-814.png,True,False,299,299,L,119.22,cc7de6b7658c4b6dc3627776ce748cad
Viral Pneumonia-815.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-815.png,Viral Pneumonia/images/Viral Pneumonia-815.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-815.png,Viral Pneumonia/masks/Viral Pneumonia-815.png,True,False,299,299,RGB,154.51,1f00e3f8fae05f9f3370f050f7772e01
Viral Pneumonia-816.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-816.png,Viral Pneumonia/images/Viral Pneumonia-816.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-816.png,Viral Pneumonia/masks/Viral Pneumonia-816.png,True,False,299,299,L,115.25,65d1caac511e5f7a439961b198fe8b05
Viral Pneumonia-817.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-817.png,Viral Pneumonia/images/Viral Pneumonia-817.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-817.png,Viral Pneumonia/masks/Viral Pneumonia-817.png,True,False,299,299,RGB,103.15,fab411a0bdc795904799b5239d31bb22
Viral Pneumonia-818.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-818.png,Viral Pneumonia/images/Viral Pneumonia-818.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-818.png,Viral Pneumonia/masks/Viral Pneumonia-818.png,True,False,299,299,L,98.84,f6857d244f1624077f851a145fbce698
Viral Pneumonia-819.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-819.png,Viral Pneumonia/images/Viral Pneumonia-819.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-819.png,Viral Pneumonia/masks/Viral Pneumonia-819.png,True,False,299,299,L,127.37,0f3ca25b55074d9986943d2fa10d8a7e
Viral Pneumonia-82.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-82.png,Viral Pneumonia/images/Viral Pneumonia-82.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-82.png,Viral Pneumonia/masks/Viral Pneumonia-82.png,True,False,299,299,L,142.53,195a24eb6985a65c990c017f90958eda
Viral Pneumonia-820.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-820.png,Viral Pneumonia/images/Viral Pneumonia-820.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-820.png,Viral Pneumonia/masks/Viral Pneumonia-820.png,True,False,299,299,L,146.91,6f08c1d81ce961cc796d146b1978ce97
Viral Pneumonia-821.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-821.png,Viral Pneumonia/images/Viral Pneumonia-821.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-821.png,Viral Pneumonia/masks/Viral Pneumonia-821.png,True,False,299,299,L,101.97,cad143d1d209e9f7cb8dc6ca5c1f1172
Viral Pneumonia-822.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-822.png,Viral Pneumonia/images/Viral Pneumonia-822.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-822.png,Viral Pneumonia/masks/Viral Pneumonia-822.png,True,False,299,299,L,119.48,f9420734d06d6703855eae138bb84256
Viral Pneumonia-823.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-823.png,Viral Pneumonia/images/Viral Pneumonia-823.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-823.png,Viral Pneumonia/masks/Viral Pneumonia-823.png,True,False,299,299,L,130.81,8a305e9f4f86228a380940149557f335
Viral Pneumonia-824.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-824.png,Viral Pneumonia/images/Viral Pneumonia-824.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-824.png,Viral Pneumonia/masks/Viral Pneumonia-824.png,True,False,299,299,L,125.85,34d83a29c68468421e6e63579cc9fc96
Viral Pneumonia-825.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-825.png,Viral Pneumonia/images/Viral Pneumonia-825.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-825.png,Viral Pneumonia/masks/Viral Pneumonia-825.png,True,False,299,299,L,122.57,51933f6331ba4c7bef57438439166cd9
Viral Pneumonia-826.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-826.png,Viral Pneumonia/images/Viral Pneumonia-826.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-826.png,Viral Pneumonia/masks/Viral Pneumonia-826.png,True,False,299,299,RGB,115.81,e5dd722a93fd06eb7b8cbd390984d710
Viral Pneumonia-827.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-827.png,Viral Pneumonia/images/Viral Pneumonia-827.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-827.png,Viral Pneumonia/masks/Viral Pneumonia-827.png,True,False,299,299,L,131.98,d529e626da16e0e8cfc120e7430ee03d
Viral Pneumonia-828.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-828.png,Viral Pneumonia/images/Viral Pneumonia-828.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-828.png,Viral Pneumonia/masks/Viral Pneumonia-828.png,True,False,299,299,L,115.62,0fd6d583d3ff16b0374339658dd42b6f
Viral Pneumonia-829.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-829.png,Viral Pneumonia/images/Viral Pneumonia-829.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-829.png,Viral Pneumonia/masks/Viral Pneumonia-829.png,True,False,299,299,L,137.39,998522c3cd75a080d58c4effdec99568
Viral Pneumonia-83.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-83.png,Viral Pneumonia/images/Viral Pneumonia-83.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-83.png,Viral Pneumonia/masks/Viral Pneumonia-83.png,True,False,299,299,L,107.87,6ca4ce32688f283081064f6540d692fa
Viral Pneumonia-830.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-830.png,Viral Pneumonia/images/Viral Pneumonia-830.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-830.png,Viral Pneumonia/masks/Viral Pneumonia-830.png,True,False,299,299,L,125.07,0a34e959933755100f282a4c2748ded2
Viral Pneumonia-831.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-831.png,Viral Pneumonia/images/Viral Pneumonia-831.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-831.png,Viral Pneumonia/masks/Viral Pneumonia-831.png,True,False,299,299,L,134.91,dd03ea29b05ad27893e8452dd28094d6
Viral Pneumonia-832.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-832.png,Viral Pneumonia/images/Viral Pneumonia-832.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-832.png,Viral Pneumonia/masks/Viral Pneumonia-832.png,True,False,299,299,L,90.87,ee34b32e9c957c70b147fe74a4997cd6
Viral Pneumonia-833.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-833.png,Viral Pneumonia/images/Viral Pneumonia-833.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-833.png,Viral Pneumonia/masks/Viral Pneumonia-833.png,True,False,299,299,L,134.34,1de9dec58b438d361fbfb8d77c90f7e0
Viral Pneumonia-834.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-834.png,Viral Pneumonia/images/Viral Pneumonia-834.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-834.png,Viral Pneumonia/masks/Viral Pneumonia-834.png,True,False,299,299,L,142.75,b939d87b572f15ea6dff2a8f9db83114
Viral Pneumonia-835.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-835.png,Viral Pneumonia/images/Viral Pneumonia-835.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-835.png,Viral Pneumonia/masks/Viral Pneumonia-835.png,True,False,299,299,L,129.37,dae715f9fdd3fe5e55425b37a7453d02
Viral Pneumonia-836.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-836.png,Viral Pneumonia/images/Viral Pneumonia-836.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-836.png,Viral Pneumonia/masks/Viral Pneumonia-836.png,True,False,299,299,L,132.12,3360e68ae245f87a1c1ed3a1f1211540
Viral Pneumonia-837.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-837.png,Viral Pneumonia/images/Viral Pneumonia-837.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-837.png,Viral Pneumonia/masks/Viral Pneumonia-837.png,True,False,299,299,L,118.34,7f0ff4a1030c1950c70c4363e6cf5c75
Viral Pneumonia-838.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-838.png,Viral Pneumonia/images/Viral Pneumonia-838.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-838.png,Viral Pneumonia/masks/Viral Pneumonia-838.png,True,False,299,299,L,131.41,b9b07a075b07c8671065cbfc9bd6cc42
Viral Pneumonia-839.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-839.png,Viral Pneumonia/images/Viral Pneumonia-839.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-839.png,Viral Pneumonia/masks/Viral Pneumonia-839.png,True,False,299,299,L,111.85,afe89e0c6d349d751ec2ea4fd3c94601
Viral Pneumonia-84.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-84.png,Viral Pneumonia/images/Viral Pneumonia-84.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-84.png,Viral Pneumonia/masks/Viral Pneumonia-84.png,True,False,299,299,L,140.19,9e1cc2f1b637296174348ce6106959b2
Viral Pneumonia-840.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-840.png,Viral Pneumonia/images/Viral Pneumonia-840.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-840.png,Viral Pneumonia/masks/Viral Pneumonia-840.png,True,False,299,299,L,119.27,6671ec5ed3cf6355fb9eb632f8602c5c
Viral Pneumonia-841.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-841.png,Viral Pneumonia/images/Viral Pneumonia-841.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-841.png,Viral Pneumonia/masks/Viral Pneumonia-841.png,True,False,299,299,L,148.8,09063ec35f736da587ec3fc243baf385
Viral Pneumonia-842.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-842.png,Viral Pneumonia/images/Viral Pneumonia-842.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-842.png,Viral Pneumonia/masks/Viral Pneumonia-842.png,True,False,299,299,L,119.5,3c1b08806f4ff61db7467e7575ab9591
Viral Pneumonia-843.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-843.png,Viral Pneumonia/images/Viral Pneumonia-843.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-843.png,Viral Pneumonia/masks/Viral Pneumonia-843.png,True,False,299,299,L,111.83,72ab5bf44c68b6f57aa2ab04eaa34a97
Viral Pneumonia-844.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-844.png,Viral Pneumonia/images/Viral Pneumonia-844.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-844.png,Viral Pneumonia/masks/Viral Pneumonia-844.png,True,False,299,299,L,134.98,242b84fd6f3a739ec7d7a9c5fcf3f616
Viral Pneumonia-845.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-845.png,Viral Pneumonia/images/Viral Pneumonia-845.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-845.png,Viral Pneumonia/masks/Viral Pneumonia-845.png,True,False,299,299,L,115.48,18a29656767e12735f3633f5ec9f719e
Viral Pneumonia-846.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-846.png,Viral Pneumonia/images/Viral Pneumonia-846.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-846.png,Viral Pneumonia/masks/Viral Pneumonia-846.png,True,False,299,299,L,129.03,8fae6c8d3cc002dd3131638c3c641f77
Viral Pneumonia-847.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-847.png,Viral Pneumonia/images/Viral Pneumonia-847.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-847.png,Viral Pneumonia/masks/Viral Pneumonia-847.png,True,False,299,299,L,117.27,bffe0486efc5d08761ca66cbef27afed
Viral Pneumonia-848.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-848.png,Viral Pneumonia/images/Viral Pneumonia-848.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-848.png,Viral Pneumonia/masks/Viral Pneumonia-848.png,True,False,299,299,L,149.48,a63dd9488c1a4c7cc06f7a86950cc64d
Viral Pneumonia-849.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-849.png,Viral Pneumonia/images/Viral Pneumonia-849.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-849.png,Viral Pneumonia/masks/Viral Pneumonia-849.png,True,False,299,299,L,107.08,cd474afd231ba9e135576d51df2cb7c2
Viral Pneumonia-85.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-85.png,Viral Pneumonia/images/Viral Pneumonia-85.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-85.png,Viral Pneumonia/masks/Viral Pneumonia-85.png,True,False,299,299,L,133.77,cd6ec754948ed0646a090b51b7ced329
Viral Pneumonia-850.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-850.png,Viral Pneumonia/images/Viral Pneumonia-850.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-850.png,Viral Pneumonia/masks/Viral Pneumonia-850.png,True,False,299,299,L,130.03,0e871c30a3e33b89dc0834d7d2887869
Viral Pneumonia-851.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-851.png,Viral Pneumonia/images/Viral Pneumonia-851.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-851.png,Viral Pneumonia/masks/Viral Pneumonia-851.png,True,False,299,299,L,138.09,c00aee71d053c640e5b920ccee8beaab
Viral Pneumonia-852.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-852.png,Viral Pneumonia/images/Viral Pneumonia-852.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-852.png,Viral Pneumonia/masks/Viral Pneumonia-852.png,True,False,299,299,L,114.86,33b24d8700096f9cd90b8f0506b99a99
Viral Pneumonia-853.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-853.png,Viral Pneumonia/images/Viral Pneumonia-853.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-853.png,Viral Pneumonia/masks/Viral Pneumonia-853.png,True,False,299,299,L,135.21,7d45ad6fca06c7829316021e5ac6f97e
Viral Pneumonia-854.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-854.png,Viral Pneumonia/images/Viral Pneumonia-854.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-854.png,Viral Pneumonia/masks/Viral Pneumonia-854.png,True,False,299,299,L,109.82,820a4ed489f2de6eaa56ee086a7597b0
Viral Pneumonia-855.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-855.png,Viral Pneumonia/images/Viral Pneumonia-855.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-855.png,Viral Pneumonia/masks/Viral Pneumonia-855.png,True,False,299,299,L,124.74,55f8b5b4005413f010cd779e7bf1d27a
Viral Pneumonia-856.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-856.png,Viral Pneumonia/images/Viral Pneumonia-856.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-856.png,Viral Pneumonia/masks/Viral Pneumonia-856.png,True,False,299,299,RGB,158.43,b5e61ae943e56bc7a3abc9d80cb7b9e9
Viral Pneumonia-857.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-857.png,Viral Pneumonia/images/Viral Pneumonia-857.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-857.png,Viral Pneumonia/masks/Viral Pneumonia-857.png,True,False,299,299,L,128.59,8b6e11038ae3455d11aca5ac446cc384
Viral Pneumonia-858.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-858.png,Viral Pneumonia/images/Viral Pneumonia-858.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-858.png,Viral Pneumonia/masks/Viral Pneumonia-858.png,True,False,299,299,L,106.07,a549834d0ebf6e050e4b74442dd6029d
Viral Pneumonia-859.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-859.png,Viral Pneumonia/images/Viral Pneumonia-859.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-859.png,Viral Pneumonia/masks/Viral Pneumonia-859.png,True,False,299,299,L,136.22,a4144aac30f6be36dda128efb5dcf71c
Viral Pneumonia-86.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-86.png,Viral Pneumonia/images/Viral Pneumonia-86.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-86.png,Viral Pneumonia/masks/Viral Pneumonia-86.png,True,False,299,299,L,144.56,b10a2560e8b0d5d87e45f9b6f3e3c979
Viral Pneumonia-860.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-860.png,Viral Pneumonia/images/Viral Pneumonia-860.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-860.png,Viral Pneumonia/masks/Viral Pneumonia-860.png,True,False,299,299,L,122.9,6fcceffd2a3e312d86bc45ef75683627
Viral Pneumonia-861.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-861.png,Viral Pneumonia/images/Viral Pneumonia-861.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-861.png,Viral Pneumonia/masks/Viral Pneumonia-861.png,True,False,299,299,L,125.48,1aa991aaa21b85b6efab9fb3dd65ea75
Viral Pneumonia-862.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-862.png,Viral Pneumonia/images/Viral Pneumonia-862.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-862.png,Viral Pneumonia/masks/Viral Pneumonia-862.png,True,False,299,299,L,138.66,51350562d911a69dba039fe92891d7ec
Viral Pneumonia-863.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-863.png,Viral Pneumonia/images/Viral Pneumonia-863.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-863.png,Viral Pneumonia/masks/Viral Pneumonia-863.png,True,False,299,299,L,135.45,3c3d7bf2138b06c55940ea1abe82fb34
Viral Pneumonia-864.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-864.png,Viral Pneumonia/images/Viral Pneumonia-864.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-864.png,Viral Pneumonia/masks/Viral Pneumonia-864.png,True,False,299,299,L,150.99,0fa748411cf3b3d7eb5edd7563cf08d9
Viral Pneumonia-865.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-865.png,Viral Pneumonia/images/Viral Pneumonia-865.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-865.png,Viral Pneumonia/masks/Viral Pneumonia-865.png,True,False,299,299,L,123.59,9ef576384cc606be926bba198202ed5c
Viral Pneumonia-866.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-866.png,Viral Pneumonia/images/Viral Pneumonia-866.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-866.png,Viral Pneumonia/masks/Viral Pneumonia-866.png,True,False,299,299,L,120.5,335fa070a6bd3c118f3747bd06d8316b
Viral Pneumonia-867.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-867.png,Viral Pneumonia/images/Viral Pneumonia-867.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-867.png,Viral Pneumonia/masks/Viral Pneumonia-867.png,True,False,299,299,L,131.99,9d39e8a01e8c58b8b014d88f4fade86b
Viral Pneumonia-868.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-868.png,Viral Pneumonia/images/Viral Pneumonia-868.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-868.png,Viral Pneumonia/masks/Viral Pneumonia-868.png,True,False,299,299,L,130.02,04caa4269fef0295ef7e89fcea654119
Viral Pneumonia-869.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-869.png,Viral Pneumonia/images/Viral Pneumonia-869.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-869.png,Viral Pneumonia/masks/Viral Pneumonia-869.png,True,False,299,299,L,112.55,f1d1c1f15ae99463f4905e37847e3069
Viral Pneumonia-87.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-87.png,Viral Pneumonia/images/Viral Pneumonia-87.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-87.png,Viral Pneumonia/masks/Viral Pneumonia-87.png,True,False,299,299,L,110.97,74a6b0ad88e7f407f934578cb23fd704
Viral Pneumonia-870.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-870.png,Viral Pneumonia/images/Viral Pneumonia-870.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-870.png,Viral Pneumonia/masks/Viral Pneumonia-870.png,True,False,299,299,L,136.13,f81f3ff2bd327cc077c4c2d82e90b994
Viral Pneumonia-871.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-871.png,Viral Pneumonia/images/Viral Pneumonia-871.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-871.png,Viral Pneumonia/masks/Viral Pneumonia-871.png,True,False,299,299,L,92.24,58b8c7b94eed0b912098f12922bf5dd7
Viral Pneumonia-872.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-872.png,Viral Pneumonia/images/Viral Pneumonia-872.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-872.png,Viral Pneumonia/masks/Viral Pneumonia-872.png,True,False,299,299,L,94.48,ce55dde65f69230e437960569e78de53
Viral Pneumonia-873.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-873.png,Viral Pneumonia/images/Viral Pneumonia-873.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-873.png,Viral Pneumonia/masks/Viral Pneumonia-873.png,True,False,299,299,L,139.58,3e8b30aa247384af6e66d6000ad800e9
Viral Pneumonia-874.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-874.png,Viral Pneumonia/images/Viral Pneumonia-874.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-874.png,Viral Pneumonia/masks/Viral Pneumonia-874.png,True,False,299,299,L,131.14,4ff9312a3056f174932aac5747c738bb
Viral Pneumonia-875.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-875.png,Viral Pneumonia/images/Viral Pneumonia-875.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-875.png,Viral Pneumonia/masks/Viral Pneumonia-875.png,True,False,299,299,L,120.54,13bee815493f7b7354ce1b466ee23a56
Viral Pneumonia-876.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-876.png,Viral Pneumonia/images/Viral Pneumonia-876.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-876.png,Viral Pneumonia/masks/Viral Pneumonia-876.png,True,False,299,299,L,117.8,9cde2a80fa21226a3706b903fe35ad90
Viral Pneumonia-877.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-877.png,Viral Pneumonia/images/Viral Pneumonia-877.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-877.png,Viral Pneumonia/masks/Viral Pneumonia-877.png,True,False,299,299,L,114.72,9e62471acbc66262ef86a2138904ac95
Viral Pneumonia-878.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-878.png,Viral Pneumonia/images/Viral Pneumonia-878.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-878.png,Viral Pneumonia/masks/Viral Pneumonia-878.png,True,False,299,299,L,106.22,273a262d038fa37e5e8ce1bea27ce3b1
Viral Pneumonia-879.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-879.png,Viral Pneumonia/images/Viral Pneumonia-879.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-879.png,Viral Pneumonia/masks/Viral Pneumonia-879.png,True,False,299,299,L,133.35,092991ddd8610e800b16dd4d20951504
Viral Pneumonia-88.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-88.png,Viral Pneumonia/images/Viral Pneumonia-88.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-88.png,Viral Pneumonia/masks/Viral Pneumonia-88.png,True,False,299,299,L,113.36,a09528a4205d2e882ba1ed3d96f95348
Viral Pneumonia-880.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-880.png,Viral Pneumonia/images/Viral Pneumonia-880.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-880.png,Viral Pneumonia/masks/Viral Pneumonia-880.png,True,False,299,299,L,127.74,884e2c95ef26426b1f6876c212cea308
Viral Pneumonia-881.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-881.png,Viral Pneumonia/images/Viral Pneumonia-881.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-881.png,Viral Pneumonia/masks/Viral Pneumonia-881.png,True,False,299,299,L,130.75,1ec21082ef20b6213b051e726d6a3f3a
Viral Pneumonia-882.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-882.png,Viral Pneumonia/images/Viral Pneumonia-882.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-882.png,Viral Pneumonia/masks/Viral Pneumonia-882.png,True,False,299,299,L,111.73,95b053a6816b352333b4c5cdc6f3a71c
Viral Pneumonia-883.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-883.png,Viral Pneumonia/images/Viral Pneumonia-883.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-883.png,Viral Pneumonia/masks/Viral Pneumonia-883.png,True,False,299,299,L,123.22,6807ff4c944a92ad70c8e8693af2ed3e
Viral Pneumonia-884.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-884.png,Viral Pneumonia/images/Viral Pneumonia-884.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-884.png,Viral Pneumonia/masks/Viral Pneumonia-884.png,True,False,299,299,L,157.93,b2b4e44e4fe616ce6a25916985c78613
Viral Pneumonia-885.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-885.png,Viral Pneumonia/images/Viral Pneumonia-885.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-885.png,Viral Pneumonia/masks/Viral Pneumonia-885.png,True,False,299,299,L,117.96,211a63d2e2b5013e12b2d24ed4783610
Viral Pneumonia-886.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-886.png,Viral Pneumonia/images/Viral Pneumonia-886.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-886.png,Viral Pneumonia/masks/Viral Pneumonia-886.png,True,False,299,299,L,124.66,93eba687b2758b2f8cb91427b624296f
Viral Pneumonia-887.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-887.png,Viral Pneumonia/images/Viral Pneumonia-887.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-887.png,Viral Pneumonia/masks/Viral Pneumonia-887.png,True,False,299,299,L,111.13,3c83c3a6d9cdb34e46a335436f093be6
Viral Pneumonia-888.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-888.png,Viral Pneumonia/images/Viral Pneumonia-888.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-888.png,Viral Pneumonia/masks/Viral Pneumonia-888.png,True,False,299,299,L,140.28,24f5c5357776a8327421e46eb989b47f
Viral Pneumonia-889.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-889.png,Viral Pneumonia/images/Viral Pneumonia-889.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-889.png,Viral Pneumonia/masks/Viral Pneumonia-889.png,True,False,299,299,L,103.64,5ed14a86f36c862b76d35f96bc4a7412
Viral Pneumonia-89.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-89.png,Viral Pneumonia/images/Viral Pneumonia-89.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-89.png,Viral Pneumonia/masks/Viral Pneumonia-89.png,True,False,299,299,L,94.95,740e43638eef2c71f791884db478ef71
Viral Pneumonia-890.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-890.png,Viral Pneumonia/images/Viral Pneumonia-890.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-890.png,Viral Pneumonia/masks/Viral Pneumonia-890.png,True,False,299,299,L,112.28,b160ceb720fd79f835b2ca7a0de119f0
Viral Pneumonia-891.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-891.png,Viral Pneumonia/images/Viral Pneumonia-891.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-891.png,Viral Pneumonia/masks/Viral Pneumonia-891.png,True,False,299,299,L,101.73,4099a78177a2c7f14db454d7a8759389
Viral Pneumonia-892.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-892.png,Viral Pneumonia/images/Viral Pneumonia-892.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-892.png,Viral Pneumonia/masks/Viral Pneumonia-892.png,True,False,299,299,L,130.98,120796c680555d7ba867ccb49bbe51b6
Viral Pneumonia-893.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-893.png,Viral Pneumonia/images/Viral Pneumonia-893.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-893.png,Viral Pneumonia/masks/Viral Pneumonia-893.png,True,False,299,299,L,104.67,3ad737c67aa31846e561a0afad2e5bcc
Viral Pneumonia-894.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-894.png,Viral Pneumonia/images/Viral Pneumonia-894.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-894.png,Viral Pneumonia/masks/Viral Pneumonia-894.png,True,False,299,299,RGB,171.49,a15b40e79e5fbb81bb74e1b4b73bb186
Viral Pneumonia-895.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-895.png,Viral Pneumonia/images/Viral Pneumonia-895.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-895.png,Viral Pneumonia/masks/Viral Pneumonia-895.png,True,False,299,299,L,109.78,8e479638f7257fb6becbfc3341059c91
Viral Pneumonia-896.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-896.png,Viral Pneumonia/images/Viral Pneumonia-896.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-896.png,Viral Pneumonia/masks/Viral Pneumonia-896.png,True,False,299,299,L,88.39,baf322ffa57b91f776e4ad44f9a8bf30
Viral Pneumonia-897.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-897.png,Viral Pneumonia/images/Viral Pneumonia-897.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-897.png,Viral Pneumonia/masks/Viral Pneumonia-897.png,True,False,299,299,L,99.23,0ca23db698c8c9cdae03527fff793cb7
Viral Pneumonia-898.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-898.png,Viral Pneumonia/images/Viral Pneumonia-898.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-898.png,Viral Pneumonia/masks/Viral Pneumonia-898.png,True,False,299,299,RGB,132.68,43bf4ad017586506827cbca0579ece64
Viral Pneumonia-899.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-899.png,Viral Pneumonia/images/Viral Pneumonia-899.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-899.png,Viral Pneumonia/masks/Viral Pneumonia-899.png,True,False,299,299,L,123.98,86cf0d4463bd2da5cb5c852252a41d35
Viral Pneumonia-9.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-9.png,Viral Pneumonia/images/Viral Pneumonia-9.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-9.png,Viral Pneumonia/masks/Viral Pneumonia-9.png,True,False,299,299,L,116.18,0c157917f6dfa5060ff82684e3621ecf
Viral Pneumonia-90.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-90.png,Viral Pneumonia/images/Viral Pneumonia-90.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-90.png,Viral Pneumonia/masks/Viral Pneumonia-90.png,True,False,299,299,L,128.87,d9c88381c2ecb86a5e0e9f655528a989
Viral Pneumonia-900.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-900.png,Viral Pneumonia/images/Viral Pneumonia-900.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-900.png,Viral Pneumonia/masks/Viral Pneumonia-900.png,True,False,299,299,L,131.09,c95088cb5dbbac96ae32500863aaa4b0
Viral Pneumonia-901.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-901.png,Viral Pneumonia/images/Viral Pneumonia-901.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-901.png,Viral Pneumonia/masks/Viral Pneumonia-901.png,True,False,299,299,L,100.82,d3a5b18a6e62eeac34ed87447c6effa4
Viral Pneumonia-902.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-902.png,Viral Pneumonia/images/Viral Pneumonia-902.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-902.png,Viral Pneumonia/masks/Viral Pneumonia-902.png,True,False,299,299,RGB,129.93,3f9153c234a972fd70cac07cc9c1ba5b
Viral Pneumonia-903.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-903.png,Viral Pneumonia/images/Viral Pneumonia-903.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-903.png,Viral Pneumonia/masks/Viral Pneumonia-903.png,True,False,299,299,L,120.47,9dd24803898f15cd65580222162cd680
Viral Pneumonia-904.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-904.png,Viral Pneumonia/images/Viral Pneumonia-904.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-904.png,Viral Pneumonia/masks/Viral Pneumonia-904.png,True,False,299,299,L,140.18,95c5704c997b29b3e9f1902922aaaa1c
Viral Pneumonia-905.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-905.png,Viral Pneumonia/images/Viral Pneumonia-905.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-905.png,Viral Pneumonia/masks/Viral Pneumonia-905.png,True,False,299,299,L,123.56,1644bb360a97248937da5d1b642309b0
Viral Pneumonia-906.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-906.png,Viral Pneumonia/images/Viral Pneumonia-906.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-906.png,Viral Pneumonia/masks/Viral Pneumonia-906.png,True,False,299,299,L,114.95,aad71cf0004221f2847b7383897fe732
Viral Pneumonia-907.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-907.png,Viral Pneumonia/images/Viral Pneumonia-907.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-907.png,Viral Pneumonia/masks/Viral Pneumonia-907.png,True,False,299,299,L,141.37,259fbcad54cd92d17c8f344ca438ada4
Viral Pneumonia-908.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-908.png,Viral Pneumonia/images/Viral Pneumonia-908.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-908.png,Viral Pneumonia/masks/Viral Pneumonia-908.png,True,False,299,299,L,128.77,c9921ddc11276a199af740ec0d6d79a1
Viral Pneumonia-909.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-909.png,Viral Pneumonia/images/Viral Pneumonia-909.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-909.png,Viral Pneumonia/masks/Viral Pneumonia-909.png,True,False,299,299,L,148.35,8ce59c0309220b3d9c8a008d8a9593d0
Viral Pneumonia-91.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-91.png,Viral Pneumonia/images/Viral Pneumonia-91.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-91.png,Viral Pneumonia/masks/Viral Pneumonia-91.png,True,False,299,299,L,142.81,a45ff7db1d593d2e33924e178807e8a5
Viral Pneumonia-910.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-910.png,Viral Pneumonia/images/Viral Pneumonia-910.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-910.png,Viral Pneumonia/masks/Viral Pneumonia-910.png,True,False,299,299,L,115.39,b3ac9401a79f6d617e0706f0da400f6d
Viral Pneumonia-911.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-911.png,Viral Pneumonia/images/Viral Pneumonia-911.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-911.png,Viral Pneumonia/masks/Viral Pneumonia-911.png,True,False,299,299,L,137.79,52076acab05d24a7568991a96dc26f61
Viral Pneumonia-912.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-912.png,Viral Pneumonia/images/Viral Pneumonia-912.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-912.png,Viral Pneumonia/masks/Viral Pneumonia-912.png,True,False,299,299,L,135.61,dc187b3d4163b7f48f144bc13b14ee19
Viral Pneumonia-913.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-913.png,Viral Pneumonia/images/Viral Pneumonia-913.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-913.png,Viral Pneumonia/masks/Viral Pneumonia-913.png,True,False,299,299,L,131.91,2c03c212c71d0e93275f1e5876d75c9e
Viral Pneumonia-914.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-914.png,Viral Pneumonia/images/Viral Pneumonia-914.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-914.png,Viral Pneumonia/masks/Viral Pneumonia-914.png,True,False,299,299,L,145.48,e434d830fa95e3469def6ffbe55f1205
Viral Pneumonia-915.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-915.png,Viral Pneumonia/images/Viral Pneumonia-915.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-915.png,Viral Pneumonia/masks/Viral Pneumonia-915.png,True,False,299,299,L,141.44,6ff16370b9a72cdbdf0722ce56213e15
Viral Pneumonia-916.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-916.png,Viral Pneumonia/images/Viral Pneumonia-916.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-916.png,Viral Pneumonia/masks/Viral Pneumonia-916.png,True,False,299,299,L,111.62,ff8f67cccb588009d5cd591220b60ddb
Viral Pneumonia-917.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-917.png,Viral Pneumonia/images/Viral Pneumonia-917.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-917.png,Viral Pneumonia/masks/Viral Pneumonia-917.png,True,False,299,299,L,115.25,1f189a5244b66882c812895ef4c6cd42
Viral Pneumonia-918.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-918.png,Viral Pneumonia/images/Viral Pneumonia-918.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-918.png,Viral Pneumonia/masks/Viral Pneumonia-918.png,True,False,299,299,RGB,143.32,d6fb65b2f4aa08f9e144f7def25408e9
Viral Pneumonia-919.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-919.png,Viral Pneumonia/images/Viral Pneumonia-919.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-919.png,Viral Pneumonia/masks/Viral Pneumonia-919.png,True,False,299,299,L,132.97,cd2d437b514aa131ed66d0b6c5e9a049
Viral Pneumonia-92.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-92.png,Viral Pneumonia/images/Viral Pneumonia-92.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-92.png,Viral Pneumonia/masks/Viral Pneumonia-92.png,True,False,299,299,L,131.22,a3f53f0f9077a3e36b4fa3267c833168
Viral Pneumonia-920.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-920.png,Viral Pneumonia/images/Viral Pneumonia-920.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-920.png,Viral Pneumonia/masks/Viral Pneumonia-920.png,True,False,299,299,RGB,138.1,867a9447896772b9ba15828b356b69ff
Viral Pneumonia-921.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-921.png,Viral Pneumonia/images/Viral Pneumonia-921.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-921.png,Viral Pneumonia/masks/Viral Pneumonia-921.png,True,False,299,299,RGB,144.06,eb4849e88d0ac5ef1957f6e20b98e327
Viral Pneumonia-922.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-922.png,Viral Pneumonia/images/Viral Pneumonia-922.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-922.png,Viral Pneumonia/masks/Viral Pneumonia-922.png,True,False,299,299,L,116.88,c2d3af3ccbc15678b2266ac32b721b7b
Viral Pneumonia-923.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-923.png,Viral Pneumonia/images/Viral Pneumonia-923.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-923.png,Viral Pneumonia/masks/Viral Pneumonia-923.png,True,False,299,299,L,85.28,aba1c954a270b33e30aa738233dd8f15
Viral Pneumonia-924.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-924.png,Viral Pneumonia/images/Viral Pneumonia-924.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-924.png,Viral Pneumonia/masks/Viral Pneumonia-924.png,True,False,299,299,L,100.95,e7860b6abdcf8607f2f3a2251e31c69a
Viral Pneumonia-925.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-925.png,Viral Pneumonia/images/Viral Pneumonia-925.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-925.png,Viral Pneumonia/masks/Viral Pneumonia-925.png,True,False,299,299,L,120.47,60f419fd333d150d6cf9c8fe88906688
Viral Pneumonia-926.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-926.png,Viral Pneumonia/images/Viral Pneumonia-926.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-926.png,Viral Pneumonia/masks/Viral Pneumonia-926.png,True,False,299,299,L,108.68,4a8099e636b352b60f5f107a1300f03d
Viral Pneumonia-927.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-927.png,Viral Pneumonia/images/Viral Pneumonia-927.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-927.png,Viral Pneumonia/masks/Viral Pneumonia-927.png,True,False,299,299,L,90.09,a34ac874191a7af5004ae7c09f9c1da6
Viral Pneumonia-928.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-928.png,Viral Pneumonia/images/Viral Pneumonia-928.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-928.png,Viral Pneumonia/masks/Viral Pneumonia-928.png,True,False,299,299,L,134.46,a352668ad44521cb451e233e7ccb9e4b
Viral Pneumonia-929.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-929.png,Viral Pneumonia/images/Viral Pneumonia-929.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-929.png,Viral Pneumonia/masks/Viral Pneumonia-929.png,True,False,299,299,RGB,105.41,60404d9bd197871299611792245d48c8
Viral Pneumonia-93.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-93.png,Viral Pneumonia/images/Viral Pneumonia-93.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-93.png,Viral Pneumonia/masks/Viral Pneumonia-93.png,True,False,299,299,L,135.5,903dd0757f892aa8c0c4479c7b8479cc
Viral Pneumonia-930.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-930.png,Viral Pneumonia/images/Viral Pneumonia-930.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-930.png,Viral Pneumonia/masks/Viral Pneumonia-930.png,True,False,299,299,L,157.82,7d814b4a97b23cb75c0893d3d93c4a51
Viral Pneumonia-931.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-931.png,Viral Pneumonia/images/Viral Pneumonia-931.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-931.png,Viral Pneumonia/masks/Viral Pneumonia-931.png,True,False,299,299,L,106.47,3e229457ee06e317a6211ed503873a70
Viral Pneumonia-932.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-932.png,Viral Pneumonia/images/Viral Pneumonia-932.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-932.png,Viral Pneumonia/masks/Viral Pneumonia-932.png,True,False,299,299,L,127.06,dbc8586899f9b3dd32840443fb7df100
Viral Pneumonia-933.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-933.png,Viral Pneumonia/images/Viral Pneumonia-933.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-933.png,Viral Pneumonia/masks/Viral Pneumonia-933.png,True,False,299,299,L,130.22,d380517e9b5a455f9cb24bcb8328b1fd
Viral Pneumonia-934.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-934.png,Viral Pneumonia/images/Viral Pneumonia-934.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-934.png,Viral Pneumonia/masks/Viral Pneumonia-934.png,True,False,299,299,L,129.97,caf1bb461e41552d75ce0e4c9db4dc60
Viral Pneumonia-935.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-935.png,Viral Pneumonia/images/Viral Pneumonia-935.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-935.png,Viral Pneumonia/masks/Viral Pneumonia-935.png,True,False,299,299,L,126.21,b9587bbbd0ac3a5960361ebcc11e4efd
Viral Pneumonia-936.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-936.png,Viral Pneumonia/images/Viral Pneumonia-936.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-936.png,Viral Pneumonia/masks/Viral Pneumonia-936.png,True,False,299,299,L,136.83,b7f298a9e2297b76c25a9760dfbf987e
Viral Pneumonia-937.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-937.png,Viral Pneumonia/images/Viral Pneumonia-937.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-937.png,Viral Pneumonia/masks/Viral Pneumonia-937.png,True,False,299,299,L,138.62,311609f949eb7956e54910ac491df7ef
Viral Pneumonia-938.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-938.png,Viral Pneumonia/images/Viral Pneumonia-938.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-938.png,Viral Pneumonia/masks/Viral Pneumonia-938.png,True,False,299,299,L,156.34,62b9e0d37f836425cb82f43c26d174fa
Viral Pneumonia-939.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-939.png,Viral Pneumonia/images/Viral Pneumonia-939.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-939.png,Viral Pneumonia/masks/Viral Pneumonia-939.png,True,False,299,299,L,130.77,60267315452b8a6f1e95bb966a7bbf0f
Viral Pneumonia-94.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-94.png,Viral Pneumonia/images/Viral Pneumonia-94.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-94.png,Viral Pneumonia/masks/Viral Pneumonia-94.png,True,False,299,299,L,144.54,5bba48eb14aa2810bbd7916558f0c4be
Viral Pneumonia-940.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-940.png,Viral Pneumonia/images/Viral Pneumonia-940.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-940.png,Viral Pneumonia/masks/Viral Pneumonia-940.png,True,False,299,299,L,115.32,c6378cf937df123b0a078ed12e0ffb3a
Viral Pneumonia-941.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-941.png,Viral Pneumonia/images/Viral Pneumonia-941.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-941.png,Viral Pneumonia/masks/Viral Pneumonia-941.png,True,False,299,299,L,129.65,354fcfa5c724b6c0816203e12ec1625e
Viral Pneumonia-942.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-942.png,Viral Pneumonia/images/Viral Pneumonia-942.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-942.png,Viral Pneumonia/masks/Viral Pneumonia-942.png,True,False,299,299,L,115.26,736a1663201aab8b8c623de7e8a6ca50
Viral Pneumonia-943.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-943.png,Viral Pneumonia/images/Viral Pneumonia-943.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-943.png,Viral Pneumonia/masks/Viral Pneumonia-943.png,True,False,299,299,L,121.04,9838de439bc4a0a09094180d3e0ea1a5
Viral Pneumonia-944.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-944.png,Viral Pneumonia/images/Viral Pneumonia-944.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-944.png,Viral Pneumonia/masks/Viral Pneumonia-944.png,True,False,299,299,L,113.26,d9bd72e4c89efc5681994fe703a6b5b5
Viral Pneumonia-945.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-945.png,Viral Pneumonia/images/Viral Pneumonia-945.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-945.png,Viral Pneumonia/masks/Viral Pneumonia-945.png,True,False,299,299,L,108.19,c56c912c6771ec412d73f21aa4c36070
Viral Pneumonia-946.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-946.png,Viral Pneumonia/images/Viral Pneumonia-946.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-946.png,Viral Pneumonia/masks/Viral Pneumonia-946.png,True,False,299,299,L,103.14,4b28c08ff9176675f528be7baa9bf726
Viral Pneumonia-947.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-947.png,Viral Pneumonia/images/Viral Pneumonia-947.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-947.png,Viral Pneumonia/masks/Viral Pneumonia-947.png,True,False,299,299,L,125.75,96208f6c4eabedfaaae723a1938240ab
Viral Pneumonia-948.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-948.png,Viral Pneumonia/images/Viral Pneumonia-948.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-948.png,Viral Pneumonia/masks/Viral Pneumonia-948.png,True,False,299,299,L,95.36,2028543f677e9614e593e392d37379fd
Viral Pneumonia-949.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-949.png,Viral Pneumonia/images/Viral Pneumonia-949.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-949.png,Viral Pneumonia/masks/Viral Pneumonia-949.png,True,False,299,299,L,92.23,1e8d4249a2a63e43564119f699c34595
Viral Pneumonia-95.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-95.png,Viral Pneumonia/images/Viral Pneumonia-95.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-95.png,Viral Pneumonia/masks/Viral Pneumonia-95.png,True,False,299,299,L,87.69,d3f63d8d130037c1d32cd7234d053c4d
Viral Pneumonia-950.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-950.png,Viral Pneumonia/images/Viral Pneumonia-950.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-950.png,Viral Pneumonia/masks/Viral Pneumonia-950.png,True,False,299,299,L,116.76,f60450b3ff6010288691bfec62600c35
Viral Pneumonia-951.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-951.png,Viral Pneumonia/images/Viral Pneumonia-951.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-951.png,Viral Pneumonia/masks/Viral Pneumonia-951.png,True,False,299,299,L,147.58,6d7d4474f65c70c3f7d8e117a5461cef
Viral Pneumonia-952.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-952.png,Viral Pneumonia/images/Viral Pneumonia-952.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-952.png,Viral Pneumonia/masks/Viral Pneumonia-952.png,True,False,299,299,L,104.95,b6749b4fdba4174e1f1227420e015170
Viral Pneumonia-953.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-953.png,Viral Pneumonia/images/Viral Pneumonia-953.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-953.png,Viral Pneumonia/masks/Viral Pneumonia-953.png,True,False,299,299,L,140.5,f3ad695e63f644e5e1347a0b6209ff22
Viral Pneumonia-954.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-954.png,Viral Pneumonia/images/Viral Pneumonia-954.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-954.png,Viral Pneumonia/masks/Viral Pneumonia-954.png,True,False,299,299,L,140.5,f3ad695e63f644e5e1347a0b6209ff22
Viral Pneumonia-955.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-955.png,Viral Pneumonia/images/Viral Pneumonia-955.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-955.png,Viral Pneumonia/masks/Viral Pneumonia-955.png,True,False,299,299,RGB,149.39,264b8898d16973c8203d6b934d97453f
Viral Pneumonia-956.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-956.png,Viral Pneumonia/images/Viral Pneumonia-956.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-956.png,Viral Pneumonia/masks/Viral Pneumonia-956.png,True,False,299,299,L,109.48,2d2bb5238d3b292f155e399eda5dc580
Viral Pneumonia-957.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-957.png,Viral Pneumonia/images/Viral Pneumonia-957.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-957.png,Viral Pneumonia/masks/Viral Pneumonia-957.png,True,False,299,299,L,109.1,b914c647d7e18227adf483d5edcf7076
Viral Pneumonia-958.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-958.png,Viral Pneumonia/images/Viral Pneumonia-958.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-958.png,Viral Pneumonia/masks/Viral Pneumonia-958.png,True,False,299,299,L,97.91,1f32529ffafdd2c7b57b64bf5b811d62
Viral Pneumonia-959.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-959.png,Viral Pneumonia/images/Viral Pneumonia-959.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-959.png,Viral Pneumonia/masks/Viral Pneumonia-959.png,True,False,299,299,L,138.62,95d9ab2b16b7230e43c4b66e26225945
Viral Pneumonia-96.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-96.png,Viral Pneumonia/images/Viral Pneumonia-96.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-96.png,Viral Pneumonia/masks/Viral Pneumonia-96.png,True,False,299,299,L,141.45,ba6d0a3398763fdfa2bad587f3db48e4
Viral Pneumonia-960.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-960.png,Viral Pneumonia/images/Viral Pneumonia-960.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-960.png,Viral Pneumonia/masks/Viral Pneumonia-960.png,True,False,299,299,L,123.46,fa9491c582ed4995dc17a3a2efb7b511
Viral Pneumonia-961.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-961.png,Viral Pneumonia/images/Viral Pneumonia-961.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-961.png,Viral Pneumonia/masks/Viral Pneumonia-961.png,True,False,299,299,L,103.75,79c8ba1f406a111b39c45260a58ba383
Viral Pneumonia-962.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-962.png,Viral Pneumonia/images/Viral Pneumonia-962.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-962.png,Viral Pneumonia/masks/Viral Pneumonia-962.png,True,False,299,299,L,110.89,f6e857d392263f3fb7adfe6a0543841b
Viral Pneumonia-963.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-963.png,Viral Pneumonia/images/Viral Pneumonia-963.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-963.png,Viral Pneumonia/masks/Viral Pneumonia-963.png,True,False,299,299,L,128.9,9919753c0e7a85f4622671349b8c95b2
Viral Pneumonia-964.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-964.png,Viral Pneumonia/images/Viral Pneumonia-964.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-964.png,Viral Pneumonia/masks/Viral Pneumonia-964.png,True,False,299,299,L,146.66,2e377003bc77f1e91a45e41ca427d747
Viral Pneumonia-965.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-965.png,Viral Pneumonia/images/Viral Pneumonia-965.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-965.png,Viral Pneumonia/masks/Viral Pneumonia-965.png,True,False,299,299,L,110.75,863c63af43a6513f89f8228c4bf5ae9f
Viral Pneumonia-966.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-966.png,Viral Pneumonia/images/Viral Pneumonia-966.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-966.png,Viral Pneumonia/masks/Viral Pneumonia-966.png,True,False,299,299,RGB,124.0,e8b71beda5154d7114c2fec9efa5b2fa
Viral Pneumonia-967.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-967.png,Viral Pneumonia/images/Viral Pneumonia-967.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-967.png,Viral Pneumonia/masks/Viral Pneumonia-967.png,True,False,299,299,RGB,137.53,863f7b64baf9f3988bede981ccb66f26
Viral Pneumonia-968.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-968.png,Viral Pneumonia/images/Viral Pneumonia-968.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-968.png,Viral Pneumonia/masks/Viral Pneumonia-968.png,True,False,299,299,L,117.13,b725012a4bf870a715c78e06e28cb940
Viral Pneumonia-969.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-969.png,Viral Pneumonia/images/Viral Pneumonia-969.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-969.png,Viral Pneumonia/masks/Viral Pneumonia-969.png,True,False,299,299,L,117.99,e62df262598e75f341d4d6a5810507ec
Viral Pneumonia-97.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-97.png,Viral Pneumonia/images/Viral Pneumonia-97.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-97.png,Viral Pneumonia/masks/Viral Pneumonia-97.png,True,False,299,299,L,121.11,f5bed6c721a9b7e89c3093385779b212
Viral Pneumonia-970.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-970.png,Viral Pneumonia/images/Viral Pneumonia-970.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-970.png,Viral Pneumonia/masks/Viral Pneumonia-970.png,True,False,299,299,RGB,113.38,2a547e7d108ee876b04797cd89680471
Viral Pneumonia-971.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-971.png,Viral Pneumonia/images/Viral Pneumonia-971.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-971.png,Viral Pneumonia/masks/Viral Pneumonia-971.png,True,False,299,299,RGB,136.55,0e8533f05fb54084f382100f21b20cb0
Viral Pneumonia-972.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-972.png,Viral Pneumonia/images/Viral Pneumonia-972.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-972.png,Viral Pneumonia/masks/Viral Pneumonia-972.png,True,False,299,299,L,108.73,ec68775e4b04d01d5f21fe09322a3153
Viral Pneumonia-973.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-973.png,Viral Pneumonia/images/Viral Pneumonia-973.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-973.png,Viral Pneumonia/masks/Viral Pneumonia-973.png,True,False,299,299,L,117.68,cfa6151f580ec8bfbe748cab12a312eb
Viral Pneumonia-974.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-974.png,Viral Pneumonia/images/Viral Pneumonia-974.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-974.png,Viral Pneumonia/masks/Viral Pneumonia-974.png,True,False,299,299,L,118.29,dd777cc89e676b75f52c097ad9807b8e
Viral Pneumonia-975.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-975.png,Viral Pneumonia/images/Viral Pneumonia-975.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-975.png,Viral Pneumonia/masks/Viral Pneumonia-975.png,True,False,299,299,L,105.67,67aede6714b69e6594ff2ad3049bd131
Viral Pneumonia-976.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-976.png,Viral Pneumonia/images/Viral Pneumonia-976.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-976.png,Viral Pneumonia/masks/Viral Pneumonia-976.png,True,False,299,299,L,130.73,53aada94db3a9991533f064c96c7ab3a
Viral Pneumonia-977.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-977.png,Viral Pneumonia/images/Viral Pneumonia-977.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-977.png,Viral Pneumonia/masks/Viral Pneumonia-977.png,True,False,299,299,L,133.5,5c59ddd1e7956c90fd9b072341b15bd4
Viral Pneumonia-978.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-978.png,Viral Pneumonia/images/Viral Pneumonia-978.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-978.png,Viral Pneumonia/masks/Viral Pneumonia-978.png,True,False,299,299,L,101.06,4563c8c36ed4161d7a1065b254a9ff11
Viral Pneumonia-979.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-979.png,Viral Pneumonia/images/Viral Pneumonia-979.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-979.png,Viral Pneumonia/masks/Viral Pneumonia-979.png,True,False,299,299,RGB,127.31,bbd303046f45c6421fa6fa7b23676368
Viral Pneumonia-98.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-98.png,Viral Pneumonia/images/Viral Pneumonia-98.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-98.png,Viral Pneumonia/masks/Viral Pneumonia-98.png,True,False,299,299,L,110.68,623be48ea28f1c875af79eff8d757051
Viral Pneumonia-980.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-980.png,Viral Pneumonia/images/Viral Pneumonia-980.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-980.png,Viral Pneumonia/masks/Viral Pneumonia-980.png,True,False,299,299,L,102.11,800c0e8923715b37b8c47ae017f5772f
Viral Pneumonia-981.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-981.png,Viral Pneumonia/images/Viral Pneumonia-981.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-981.png,Viral Pneumonia/masks/Viral Pneumonia-981.png,True,False,299,299,RGB,144.33,0e7523dd99034a8da7d6f45b0a7fdcc0
Viral Pneumonia-982.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-982.png,Viral Pneumonia/images/Viral Pneumonia-982.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-982.png,Viral Pneumonia/masks/Viral Pneumonia-982.png,True,False,299,299,RGB,138.11,9d3893eb20389d484251ffef02fa0b3d
Viral Pneumonia-983.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-983.png,Viral Pneumonia/images/Viral Pneumonia-983.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-983.png,Viral Pneumonia/masks/Viral Pneumonia-983.png,True,False,299,299,L,117.24,62bbc965bf8c8fc6d640b1f6ae98ddbd
Viral Pneumonia-984.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-984.png,Viral Pneumonia/images/Viral Pneumonia-984.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-984.png,Viral Pneumonia/masks/Viral Pneumonia-984.png,True,False,299,299,L,117.31,9aec9692bc25d64c6caa711103ab263b
Viral Pneumonia-985.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-985.png,Viral Pneumonia/images/Viral Pneumonia-985.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-985.png,Viral Pneumonia/masks/Viral Pneumonia-985.png,True,False,299,299,L,74.7,2dc69d781a0d072ac5b71b4636764ce7
Viral Pneumonia-986.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-986.png,Viral Pneumonia/images/Viral Pneumonia-986.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-986.png,Viral Pneumonia/masks/Viral Pneumonia-986.png,True,False,299,299,L,132.51,ce1fea34442492b85f05cae5347f6061
Viral Pneumonia-987.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-987.png,Viral Pneumonia/images/Viral Pneumonia-987.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-987.png,Viral Pneumonia/masks/Viral Pneumonia-987.png,True,False,299,299,L,113.08,3c4c2babd5732e926790c5a8beafd91d
Viral Pneumonia-988.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-988.png,Viral Pneumonia/images/Viral Pneumonia-988.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-988.png,Viral Pneumonia/masks/Viral Pneumonia-988.png,True,False,299,299,L,74.99,5f76b3c2b79c73027ffd5e4151fe49c0
Viral Pneumonia-989.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-989.png,Viral Pneumonia/images/Viral Pneumonia-989.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-989.png,Viral Pneumonia/masks/Viral Pneumonia-989.png,True,False,299,299,L,120.36,629606c6993e91449066ab7e530a220e
Viral Pneumonia-99.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-99.png,Viral Pneumonia/images/Viral Pneumonia-99.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-99.png,Viral Pneumonia/masks/Viral Pneumonia-99.png,True,False,299,299,L,128.32,78e6e601fc379375b3107bf5a9d35c41
Viral Pneumonia-990.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-990.png,Viral Pneumonia/images/Viral Pneumonia-990.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-990.png,Viral Pneumonia/masks/Viral Pneumonia-990.png,True,False,299,299,L,85.43,80ca3f169cc373939d7c5012ff17215c
Viral Pneumonia-991.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-991.png,Viral Pneumonia/images/Viral Pneumonia-991.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-991.png,Viral Pneumonia/masks/Viral Pneumonia-991.png,True,False,299,299,L,136.12,5b52cd12a768990422dff0a07b0c8580
Viral Pneumonia-992.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-992.png,Viral Pneumonia/images/Viral Pneumonia-992.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-992.png,Viral Pneumonia/masks/Viral Pneumonia-992.png,True,False,299,299,L,135.11,7a0b1d4e2adeb3d7821d675b4b14b6d5
Viral Pneumonia-993.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-993.png,Viral Pneumonia/images/Viral Pneumonia-993.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-993.png,Viral Pneumonia/masks/Viral Pneumonia-993.png,True,False,299,299,L,119.04,36e0fb621fdd3a6936a93c06d3ee9f8c
Viral Pneumonia-994.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-994.png,Viral Pneumonia/images/Viral Pneumonia-994.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-994.png,Viral Pneumonia/masks/Viral Pneumonia-994.png,True,False,299,299,L,131.08,d06a56f63ec73c0b45e3688fc7d9a174
Viral Pneumonia-995.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-995.png,Viral Pneumonia/images/Viral Pneumonia-995.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-995.png,Viral Pneumonia/masks/Viral Pneumonia-995.png,True,False,299,299,L,120.34,3b87d27355f958edbb3ab2f79834ec10
Viral Pneumonia-996.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-996.png,Viral Pneumonia/images/Viral Pneumonia-996.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-996.png,Viral Pneumonia/masks/Viral Pneumonia-996.png,True,False,299,299,L,120.86,f5b2a4a3a3ff4efca48cb41f9ef74ce3
Viral Pneumonia-997.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-997.png,Viral Pneumonia/images/Viral Pneumonia-997.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-997.png,Viral Pneumonia/masks/Viral Pneumonia-997.png,True,False,299,299,L,118.23,06057f9a26f657073f446e0650668d5c
Viral Pneumonia-998.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-998.png,Viral Pneumonia/images/Viral Pneumonia-998.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-998.png,Viral Pneumonia/masks/Viral Pneumonia-998.png,True,False,299,299,RGB,143.85,8dc33fbac06668f6ef48e1d7ee0227f2
Viral Pneumonia-999.png,Viral Pneumonia,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images/Viral Pneumonia-999.png,Viral Pneumonia/images/Viral Pneumonia-999.png,/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/masks/Viral Pneumonia-999.png,Viral Pneumonia/masks/Viral Pneumonia-999.png,True,False,299,299,L,126.11,385992c8e0971305a95a8dc9880b1e5e
''')

print("Đã thiết lập đầy đủ mã nguồn và dữ liệu split_v1.csv, manifest.csv!")


In [ ]:
# 2. MÔ HÌNH CƠ SỞ B0: Luôn dự đoán lớp đa số của tập Train
from src.metrics import compute_metrics, DEFAULT_CLASS_NAMES
from src.utils import set_seed

set_seed(42)
split_df = pd.read_csv("splits/split_v1.csv")
train_df = split_df[split_df["split"] == "train"]
val_df = split_df[split_df["split"] == "val"]

majority_class = train_df["label"].value_counts().index[0]
majority_idx = DEFAULT_CLASS_NAMES.index(majority_class)
print(f"Lớp đa số trong tập Train: {majority_class} (chỉ số {majority_idx})")

y_true_val = [DEFAULT_CLASS_NAMES.index(lbl) for lbl in val_df["label"]]
y_pred_b0 = [majority_idx] * len(val_df)

metrics_b0 = compute_metrics(y_true_val, y_pred_b0, class_names=DEFAULT_CLASS_NAMES)

print("\n=== KẾT QUẢ MÔ HÌNH CƠ SỞ B0 TRÊN TẬP VALIDATION ===")
print(f"Accuracy          : {metrics_b0['accuracy']:.4f}")
print(f"Balanced Accuracy : {metrics_b0['balanced_accuracy']:.4f}")
print(f"Macro-F1          : {metrics_b0['macro_f1']:.4f}")
for c, vals in metrics_b0['per_class'].items():
    print(f"  Lớp {c:<16}: Recall = {vals['recall']:.4f}, Precision = {vals['precision']:.4f}, F1 = {vals['f1']:.4f}")

with open("results/b0_majority/metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_b0, f, indent=2, ensure_ascii=False)


In [ ]:
# 3. MÔ HÌNH PHI BỆNH LÝ A0: Học lối tắt từ độ sáng trung bình và kênh màu
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

manifest_df = pd.read_csv("splits/manifest.csv")
merged = split_df.merge(manifest_df[["image_id", "color_mode", "mean_brightness"]], on="image_id")

merged["is_rgb"] = (merged["color_mode"] == "RGB").astype(float)
merged["brightness"] = merged["mean_brightness"].astype(float)

train_m = merged[merged["split"] == "train"]
val_m = merged[merged["split"] == "val"]

feats = ["brightness", "is_rgb"]
X_tr = train_m[feats].values
y_tr = np.array([DEFAULT_CLASS_NAMES.index(l) for l in train_m["label"]])

X_va = val_m[feats].values
y_va = np.array([DEFAULT_CLASS_NAMES.index(l) for l in val_m["label"]])

# Huấn luyện Logistic Regression
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_va_s = scaler.transform(X_va)

clf_lr = LogisticRegression(random_state=42, max_iter=1000)
clf_lr.fit(X_tr_s, y_tr)
y_pred_a0_lr = clf_lr.predict(X_va_s)
metrics_a0_lr = compute_metrics(y_va, y_pred_a0_lr, class_names=DEFAULT_CLASS_NAMES)

# Huấn luyện Random Forest
clf_rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
clf_rf.fit(X_tr, y_tr)
y_pred_a0_rf = clf_rf.predict(X_va)
metrics_a0_rf = compute_metrics(y_va, y_pred_a0_rf, class_names=DEFAULT_CLASS_NAMES)

print("\n=== KẾT QUẢ MÔ HÌNH PHI BỆNH LÝ A0 TRÊN TẬP VALIDATION ===")
print(f"A0 (Logistic Regression) Macro-F1: {metrics_a0_lr['macro_f1']:.4f} | Balanced Acc: {metrics_a0_lr['balanced_accuracy']:.4f}")
print(f"A0 (Random Forest depth=5) Macro-F1: {metrics_a0_rf['macro_f1']:.4f} | Balanced Acc: {metrics_a0_rf['balanced_accuracy']:.4f}")

with open("results/a0_shortcut/metrics.json", "w", encoding="utf-8") as f:
    json.dump({"logistic_regression": metrics_a0_lr, "random_forest": metrics_a0_rf}, f, indent=2, ensure_ascii=False)


In [ ]:
# 4. PILOT RUN B3: ResNet18 Full Fine-tuning (Seed 42) trên GPU
# Đo lường thời gian giây/epoch, tổng thời gian run, GPU-giờ tiêu hao
import torch.nn as nn
from src.models import get_model
from src.dataset import build_dataloaders
from src.train import train_pipeline

set_seed(42)
dataset_root = None
for p in Path("/kaggle/input").rglob("COVID"):
    if p.is_dir() and (p / "images").exists():
        dataset_root = p.parent
        break
if dataset_root is None:
    dataset_root = Path("/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset")
print(f"Đường dẫn dataset: {dataset_root} (Tồn tại: {dataset_root.exists()})")

# Nạp DataLoader cho Train và Val (khóa test allow_test=False)
dataloaders = build_dataloaders(
    split_csv_path="splits/split_v1.csv",
    data_root=dataset_root,
    batch_size=32,
    num_workers=2,
    img_size=224,
    allow_test=False,
)

# Khởi tạo ResNet18 với trọng số ImageNet official (Phương án A: có internet)
print("Đang nạp mô hình ResNet18 với trọng số official ResNet18_Weights.IMAGENET1K_V1...")
model_b3 = get_model(model_name="resnet18", num_classes=4, pretrained=True)

# Thiết lập tối ưu và hàm mất mát
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model_b3.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=1)

# Chạy huấn luyện Pilot 5 epochs
run_id_pilot = "b3_resnet18_full_s42_pilot"
t_pilot_start = time.time()

pilot_res = train_pipeline(
    model=model_b3,
    train_loader=dataloaders["train"],
    val_loader=dataloaders["val"],
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    num_epochs=5,
    device_name="cuda" if torch.cuda.is_available() else "cpu",
    output_dir=Path("outputs/checkpoints"),
    run_id=run_id_pilot,
    use_amp=True,
    class_names=DEFAULT_CLASS_NAMES,
)

t_pilot_total = time.time() - t_pilot_start
gpu_hours = t_pilot_total / 3600.0

print(f"\n=== PILOT RUN B3 HOÀN TẤT ===")
print(f"Tổng thời gian: {t_pilot_total:.2f} giây ({t_pilot_total/60.0:.2f} phút)")
print(f"GPU-giờ tiêu hao: {gpu_hours:.4f} giờ GPU")
print(f"Best Val Macro-F1: {pilot_res['best_val_f1']:.4f}")

# Đánh giá chi tiết checkpoint tốt nhất trên tập Validation
from src.train import validate, load_checkpoint
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_best = get_model(model_name="resnet18", num_classes=4, pretrained=False)
load_checkpoint(Path(pilot_res['best_checkpoint']), model_best, device=device)
val_loss_best, val_metrics_best = validate(model_best, dataloaders["val"], criterion, device, class_names=DEFAULT_CLASS_NAMES)

pilot_summary = {
    "run_id": run_id_pilot,
    "model": "resnet18",
    "weights": "ResNet18_Weights.IMAGENET1K_V1",
    "strategy": "full_finetune",
    "seed": 42,
    "epochs": 5,
    "batch_size": 32,
    "total_seconds": round(t_pilot_total, 2),
    "gpu_hours": round(gpu_hours, 4),
    "best_val_f1": round(pilot_res['best_val_f1'], 4),
    "best_val_metrics": val_metrics_best,
}

with open("results/pilot_b3/pilot_summary.json", "w", encoding="utf-8") as f:
    json.dump(pilot_summary, f, indent=2, ensure_ascii=False)

print("Đã lưu kết quả tóm tắt Pilot run tại: results/pilot_b3/pilot_summary.json")
